# GEMV kernel with alpha and beta

During LLM decoding with batch size 1, every linear layer is a matrix-vector product: the weights are read once and each weight is used for only two flops, so the kernel is limited by memory bandwidth. Write the BLAS-style GEMV:

$$y_{\text{out}} = \alpha\, A x + \beta\, y$$

Write `gemv(A, x, y, alpha, beta)`.

## Inputs
- `A`: `float32` CUDA tensor `(M, N)`, row-major (use `A.stride(0)`; the column stride is 1).
- `x`: `float32` `(N,)`; `y`: `float32` `(M,)`.
- `alpha`, `beta`: Python floats.

## Output
- A new `float32` tensor `(M,)`. `y` itself is not modified.
- **BLAS rule:** when `beta == 0`, `y` is not read at all, so NaN or garbage in `y` must not reach the output.

## Requirements
- A 1-D grid of `cdiv(M, BLOCK_M)` programs; each owns `BLOCK_M` rows and loops over the columns in chunks of `BLOCK_N`.
- Each step loads a `(BLOCK_M, BLOCK_N)` tile of `A` and a `(BLOCK_N,)` chunk of `x`, multiplies with broadcasting and accumulates into a `(BLOCK_M, BLOCK_N)` `float32` accumulator. Reduce over the columns once after the loop. No `tl.dot`.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a float64 PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def gemv(A, x, y, alpha, beta):
    """A: (M, N), x: (N,), y: (M,) float32 CUDA. Return alpha * A @ x + beta * y; y unread when beta == 0."""
    raise NotImplementedError


@triton.jit
def _gemv_kernel(a_ptr, x_ptr, y_ptr, out_ptr, M, N, stride_am, alpha, beta,
                 BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, USE_Y: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0], [3.0, 4.0]], "dtype": "float32"}, {"__nd__": [1.0, 1.0], "dtype": "float32"}, {"__nd__": [10.0, 20.0], "dtype": "float32"}, 1.0, 1.0], "e": {"__nd__": [13.0, 27.0], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 0.0, 2.0]], "dtype": "float32"}, {"__nd__": [3.0, 4.0, 5.0], "dtype": "float32"}, {"__nd__": [1.0], "dtype": "float32"}, 2.0, -1.0], "e": {"__nd__": [25.0], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.231665015220642, 0.2671189606189728, -0.006926123518496752, 0.5015352964401245, -1.326728343963623, 1.107768177986145, 0.09375488013029099, -1.1708180904388428, -1.3582403659820557, -1.3065639734268188, -0.7176153063774109, 1.1856213808059692, 0.8945337533950806, -0.5440478324890137, -0.5465879440307617, 1.929587721824646, 0.9027885794639587, -0.8033905625343323, -0.1497119665145874, -0.4302915930747986, 0.15154145658016205, 0.879149854183197, -0.2748914659023285, 0.36928102374076843, 0.568265438079834, 0.7835076451301575, -0.6304340958595276, -0.221796452999115, 1.972353458404541, -0.1859680563211441, 0.029913093894720078, 0.4409482479095459, -0.5772605538368225, -0.38974758982658386, 1.6782280206680298, 1.296455979347229, -1.1034374237060547, 0.06516579538583755, -1.2894296646118164, -1.531747579574585, 0.002736472524702549, 1.5245542526245117, -0.08182613551616669, 1.470201015472412, -1.0568504333496094, 0.13626828789710999, 0.0809459462761879, 0.08619596809148788, 0.9934000968933105, 0.5733787417411804, -0.5490205883979797, -1.3512771129608154, -0.5538102984428406, -0.02058545872569084, -0.5506706833839417, -1.5250804424285889, -1.7636319398880005, 0.8112818002700806, 0.2564311623573303, -0.5855581164360046, 0.44081568717956543, 0.49441832304000854, -0.21350893378257751, -0.42714518308639526, 1.1488161087036133, 0.8532398343086243, 1.4325295686721802, 1.0122079849243164, 0.2584077715873718, -0.4491339921951294, -0.342641144990921, -0.5945685505867004, 0.06525478512048721, 0.2091742902994156, 1.1665877103805542, 1.3475943803787231, -0.018009493127465248, 0.38995879888534546, -1.1755001544952393, -0.6776876449584961, -1.0862568616867065, 0.12670397758483887, -0.5381069183349609, -0.7740989327430725, -0.02922406792640686, -1.8748778104782104, -0.09780754894018173, 0.4532788097858429, 1.23257577419281, -1.1779193878173828, -0.436354398727417, -2.0592474937438965, 1.3180829286575317, -0.21181365847587585, 0.21356536448001862, -1.539515495300293, -0.27373242378234863, -0.6360392570495605, -1.2428117990493774, -1.2089320421218872, -0.9871983528137207, -0.9083639979362488, -1.7987728118896484, 1.272772192955017, -0.0793139785528183, 0.060225922614336014, 0.044567592442035675, -0.3296782970428467, -0.18658225238323212, 0.4264450967311859, -0.3936433792114258, 1.1390490531921387, -0.3440118730068207, 1.020664930343628, -1.0355762243270874, 0.15507908165454865, 0.5550908446311951, 1.180607557296753, 1.2072468996047974, -1.0895689725875854, -0.7675547003746033, 1.4228184223175049, -0.4935930371284485, -1.621298909187317, 0.9784850478172302, 0.4722200632095337, -1.30815589427948, -0.7023630142211914, 1.1429699659347534, -0.5613574385643005, -0.6684896349906921, -0.526546835899353, 1.1640942096710205, -1.7585563659667969, 0.19440534710884094, -0.07624801248311996, 0.25920358300209045, 0.5132573246955872, -0.1721293181180954, 0.36764371395111084, 0.2298281192779541, 1.189131259918213, -2.4045605659484863, -0.3134860396385193, -0.07221069931983948, -1.4599545001983643, 1.2167854309082031, 0.3932887017726898, -1.7590194940567017, -0.35882997512817383, 0.5960685014724731, 0.025773322209715843, -0.7931777238845825, 0.5250595808029175, 0.2500934302806854, -0.9445063471794128, -0.26594141125679016, 1.2275904417037964, 1.119468092918396, -0.9631155729293823, -0.6444364190101624, 1.3431423902511597, 0.5524418354034424, 0.029327161610126495, -0.32119131088256836, 0.17749160528182983, 0.8866062164306641, -2.288200855255127, 0.829703152179718, -0.8995903730392456, -1.2850415706634521, -0.6508762240409851, -1.0716497898101807, -0.9911635518074036, 0.038889165967702866, -0.42375022172927856, 0.957984447479248, -1.5076820850372314, 0.7718095779418945, -1.1784183979034424, 0.1596258580684662, -0.32315003871917725, 0.9225702881813049, -1.2341771125793457, 0.08295951783657074, 1.5358880758285522, 1.2605599164962769, -0.2439141571521759, 0.770711362361908, 1.0872801542282104, 0.5351113677024841, 1.2129619121551514, 1.0185121297836304, -2.1618597507476807, 0.6773707866668701, 3.5597283840179443, -0.31077268719673157, -0.24353623390197754, 0.10438201576471329, -0.7267517447471619, -0.26457422971725464, -0.04359615966677666, 2.0314807891845703, -0.18332357704639435, 0.6167254447937012, -0.1390594244003296, -0.8664671778678894, -0.6007385849952698, -1.056006669998169, 1.4052516222000122, -0.19162526726722717, -1.38845694065094, 0.03138507530093193, 2.082955837249756, 0.3078670799732208, 0.008673423901200294, 0.32595303654670715, -0.34332075715065, -0.9822558760643005, 1.3391870260238647, -3.2547059059143066, -0.4892156422138214, -0.8556835055351257, -0.17551381886005402, -0.7015539407730103, 1.157409906387329, 1.5896323919296265, 0.18425005674362183, -1.2253994941711426, -0.7015854716300964, -0.32697615027427673, -0.8646931052207947, -1.2076977491378784, 1.7627272605895996, 0.9617324471473694, -0.9412595629692078, -0.851597249507904, 0.08525010943412781, -0.6428089737892151, 0.593061625957489, 0.8079827427864075, 0.06297189742326736, -0.9002277255058289, -0.29209840297698975, -1.6145751476287842, -0.3478091061115265, 1.8098320960998535, -0.5386336445808411, 0.6886866092681885, 1.2130156755447388, 1.6145192384719849, -1.3120112419128418, -0.1924862414598465, -0.5493267774581909, 1.3663653135299683, 1.0959259271621704, -1.0438508987426758, -0.5422149300575256, 0.366072416305542, 2.058976173400879, -0.35550379753112793, 1.2292213439941406, -0.8260408639907837, -1.2130968570709229, 0.8253031969070435, -0.7692468166351318, -1.5506521463394165, 1.2537946701049805, 0.6117683053016663, -1.2745925188064575, 0.857964277267456, 0.4337024390697479, 0.5588881373405457, -1.060407042503357, -0.41429799795150757, -2.1277596950531006, -2.129279851913452, 0.8037746548652649, -0.46875032782554626, 0.20520815253257751, -0.2271338701248169, -0.530676007270813, -0.5164893269538879, 0.4858003556728363, -0.24624468386173248, -1.5440070629119873, 1.1161552667617798, 1.3851897716522217, 0.7613213062286377, 1.476744532585144, 0.18322238326072693, 0.022499946877360344, 1.1285054683685303, 0.5375382304191589, -1.1878360509872437, -0.1755090206861496, 0.08813906461000443, 2.184619903564453, 0.04455455765128136, -0.7684325575828552], [-1.0834417343139648, -0.1652008444070816, -0.6478673815727234, -0.12506245076656342, -1.014116883277893, -1.6235766410827637, -0.2932146489620209, 0.33261555433273315, 0.1041238084435463, -1.536759376525879, -0.318141907453537, 0.7739880084991455, 0.2690930962562561, -0.656875491142273, -0.4777217507362366, -0.47695234417915344, 0.18829451501369476, -0.39333564043045044, 0.42789047956466675, 0.14740756154060364, 0.1303260326385498, 0.760811984539032, 0.8184022903442383, -0.41750234365463257, -0.31030595302581787, 0.8960642218589783, -0.3192208707332611, -1.2501672506332397, -0.5973032116889954, -1.0774143934249878, -0.05341658741235733, 0.07965216785669327, 1.3070718050003052, 0.401521235704422, 0.5813412666320801, 1.4593138694763184, 0.7279235124588013, -0.8917006850242615, -0.09770470857620239, -0.671313464641571, -0.21694064140319824, 1.1539034843444824, 2.8165431022644043, 0.7882928252220154, 0.22549296915531158, -0.7381506562232971, 0.24933147430419922, 1.527621865272522, -0.7919186353683472, 0.4065745174884796, 0.8057714104652405, 0.9353095889091492, 2.3535754680633545, 0.0011273984564468265, 0.2394411414861679, -1.0382267236709595, -0.4736107289791107, -0.9314520955085754, 0.08250977843999863, 0.4278387129306793, 0.6690530180931091, 0.7707390785217285, -0.511815071105957, -1.0085352659225464, -1.0098390579223633, -0.2984476089477539, -0.45233121514320374, -1.1882069110870361, 0.3851269483566284, -1.104894995689392, -0.1368810385465622, -0.6051692962646484, 0.32264477014541626, 0.47580310702323914, -0.9136974215507507, 0.40290042757987976, 1.9751986265182495, 0.44679248332977295, -1.071823239326477, 2.171954870223999, -0.9415614604949951, 0.7091088891029358, -1.0639488697052002, 0.7279537916183472, -0.15171284973621368, 0.5139342546463013, 0.22257617115974426, -1.4824752807617188, -1.5991263389587402, -0.8412506580352783, 0.8653973340988159, 0.311890184879303, 1.1964235305786133, 1.208795189857483, 1.694093942642212, -0.9437826871871948, -1.1534873247146606, -1.1008813381195068, 1.3384267091751099, -0.4076651334762573, 0.3021187484264374, 1.9075424671173096, -0.38002482056617737, -0.6118348836898804, -0.9361985325813293, 0.9660681486129761, 1.0271005630493164, 0.10220704227685928, -0.3633004128932953, -0.10827392339706421, 0.3854491412639618, 1.0662951469421387, -0.9776581525802612, -0.7572444081306458, 0.8888245820999146, 0.8397271633148193, 0.046126838773489, -0.135152667760849, 0.9932818412780762, 0.13732866942882538, -1.700393557548523, -1.0488319396972656, 0.7121216654777527, -0.16205856204032898, 0.4195471704006195, -1.7103497982025146, 1.1321625709533691, 0.24740475416183472, -0.19701114296913147, 0.07815790176391602, -0.32034990191459656, -0.22788113355636597, -2.263148307800293, -1.0202842950820923, 1.0695221424102783, -0.21817955374717712, -0.8919280171394348, 0.4798694849014282, 1.4781935214996338, -0.027863672003149986, 0.818722665309906, -0.6407896280288696, -1.7858518362045288, 1.5478484630584717, -0.7351582050323486, 0.7994324564933777, 1.0412834882736206, 1.6675444841384888, -0.5209419131278992, 1.1061100959777832, -0.9250873327255249, -1.423510193824768, -1.600173830986023, 0.22704410552978516, -1.4539506435394287, 2.6615073680877686, 1.095529317855835, 1.0720417499542236, 0.3822516202926636, 0.9474648833274841, -0.41369321942329407, -1.0259732007980347, -0.8081739544868469, 0.6282041668891907, 0.10495582967996597, 0.42119520902633667, -1.3187862634658813, 0.8374298810958862, 1.2858341932296753, 0.413628488779068, -2.2528350353240967, -0.5347747802734375, -0.002246672986075282, 1.0999835729599, 0.21350252628326416, 0.8156073689460754, -0.9464479684829712, -0.6721612811088562, -0.672268271446228, 1.3775789737701416, -1.7446649074554443, -0.40269455313682556, -1.820515513420105, 1.6473369598388672, 0.7517916560173035, -0.7952132225036621, 0.7332628965377808, 0.7240670919418335, -0.23329947888851166, 0.33332401514053345, -0.19824635982513428, -0.19442948698997498, -1.723152995109558, -0.9345338940620422, 0.7066067457199097, 0.16122467815876007, 2.172189712524414, 1.2989989519119263, 0.4529370665550232, -1.9742639064788818, -0.6613357663154602, 0.17515631020069122, 1.938994288444519, 0.44815751910209656, -1.4917690753936768, 0.8592001795768738, 1.0423330068588257, 1.2574796676635742, 1.4672811031341553, -0.8816974759101868, -0.03152452036738396, 0.5040087699890137, 0.9133619666099548, -0.6061651110649109, 0.9991356134414673, 0.8631970882415771, -2.2675278186798096, 0.6602867245674133, 0.27918583154678345, -0.6040013432502747, -0.4314268231391907, 0.4883516728878021, 1.7087360620498657, 0.5461517572402954, -1.5430269241333008, -0.42990151047706604, 0.04754157364368439, -2.4676594734191895, 0.007749617099761963, -0.709570050239563, -0.4830385744571686, -1.228315830230713, 0.3249797523021698, -1.1249394416809082, -1.1646445989608765, 0.04355262964963913, 0.7784701585769653, -0.6167070269584656, -0.408651202917099, 0.7196590900421143, 0.40929803252220154, 0.04180089011788368, 1.053951382637024, 0.7956236600875854, -1.0852587223052979, -1.140965461730957, 1.7550159692764282, -0.83924800157547, -1.9810153245925903, -0.44241058826446533, 0.02689039707183838, 0.7859211564064026, 1.8421152830123901, -0.12441020458936691, -2.1090991497039795, -0.7442079186439514, 0.5186458826065063, 1.3943296670913696, -0.3075172007083893, -0.9455418586730957, -0.4960661232471466, -1.5161741971969604, -1.1952643394470215, 2.2466249465942383, 1.6047972440719604, -0.6717226505279541, -1.291373610496521, 0.792788565158844, 0.6519691348075867, 0.4639962315559387, -1.9764232635498047, -0.1417907327413559, -0.9012150168418884, 1.0648075342178345, -2.704108953475952, 0.7642418742179871, -0.6752407550811768, 0.8838750123977661, -0.5501189827919006, 1.1547924280166626, -0.3567085862159729, -0.6903970837593079, -0.5640931129455566, 1.400327444076538, -0.8451505899429321, 0.2742101550102234, 0.8283810019493103, -1.1285778284072876, 0.04336247220635414, -0.285535603761673, 1.4970165491104126, -0.6372200846672058, 0.4623989462852478, -1.192579746246338, -1.492820143699646, 0.569758415222168, 0.7604403495788574, 0.6776551604270935, -2.8079512119293213, 1.5642465353012085], [0.4216381311416626, 0.36218154430389404, 2.22953462600708, 0.8864696025848389, 0.1661456823348999, -1.0626935958862305, -0.13737937808036804, 0.9266942739486694, -2.093247413635254, 0.34173688292503357, -0.5215632915496826, -0.7021514177322388, 1.014657974243164, -0.16986511647701263, -1.550289511680603, -0.9185180068016052, 0.473641574382782, 0.6571658849716187, -1.8590530157089233, -0.9021993279457092, 0.2999701499938965, 1.3554394245147705, 0.5923871994018555, -0.6913319230079651, 0.7222728729248047, -0.9278251528739929, 0.27428948879241943, -0.1297154575586319, 1.1342929601669312, 1.3320456743240356, 1.5369799137115479, 0.24899277091026306, -0.008509857580065727, 0.3916831314563751, 2.3567326068878174, 0.17732226848602295, 0.3716335892677307, 0.0711330845952034, -1.0175437927246094, 0.9121126532554626, 0.6972098350524902, 0.19502156972885132, -0.5165874361991882, -0.34561002254486084, -0.4185259938240051, -1.53761625289917, -0.45952078700065613, 1.1177555322647095, 0.5642425417900085, -0.30938753485679626, 0.7988415360450745, -0.2378382384777069, -0.5913987755775452, 0.0752592608332634, 0.36594414710998535, 1.8815771341323853, 0.13791990280151367, 0.07610978931188583, 0.6220738887786865, -1.3058116436004639, 1.3727638721466064, 1.8237011432647705, -0.11412079632282257, -0.5077771544456482, 0.20543436706066132, 0.39393091201782227, -0.6881393194198608, -1.5867570638656616, -0.05116407200694084, -0.642633855342865, 1.4672175645828247, -0.10704654455184937, 0.03189164400100708, 0.1457429975271225, -1.0593477487564087, -0.8966315984725952, 1.956093668937683, 1.3950798511505127, -1.1729732751846313, 0.36325037479400635, -0.28739190101623535, 0.8077475428581238, 0.9959716200828552, -0.43393391370773315, 1.0085800886154175, -0.4977903366088867, 0.3932628333568573, 1.6609418392181396, 0.638789176940918, 0.23541748523712158, 1.4902147054672241, -0.5810400247573853, 0.6496943235397339, -0.2109520435333252, -1.5696839094161987, -0.09217890352010727, 0.42429888248443604, -2.031669855117798, -1.9722496271133423, -0.1519618034362793, 0.0039885034784674644, -0.14049509167671204, 1.6914026737213135, 3.1140687465667725, -0.27312904596328735, 0.2836156487464905, -0.6187319159507751, 1.126962661743164, -2.01898455619812, 1.1500650644302368, -1.8375786542892456, 0.34696462750434875, 0.44830450415611267, -1.0315192937850952, 0.18614009022712708, 1.5143280029296875, -0.06298906356096268, 0.25813764333724976, -1.1969043016433716, -0.3685195744037628, -1.6031523942947388, 1.4479721784591675, -0.2587984800338745, 1.0410621166229248, 0.9751161932945251, -0.3735150396823883, 0.6021662354469299, 1.087123155593872, 1.3625839948654175, -2.87687349319458, 1.5308241844177246, 1.4033501148223877, 0.26044443249702454, -0.13656561076641083, 0.16203653812408447, 0.8698691129684448, -1.9424467086791992, 1.8131564855575562, -1.2186380624771118, 2.3673489093780518, 0.45873159170150757, 1.141319751739502, 0.3470524251461029, -0.44462504982948303, -1.3321189880371094, -0.6492195725440979, 0.9521651268005371, -0.3815426230430603, -0.03368789330124855, -0.11622872203588486, -0.8186551928520203, -0.5016334652900696, 1.4846855401992798, 1.4823263883590698, 2.8692660331726074, -0.8747361898422241, -1.0838700532913208, 0.17127788066864014, -1.4571906328201294, 0.3768921494483948, 1.7143133878707886, 0.41170263290405273, -0.31131434440612793, -1.8833664655685425, -0.010162707418203354, -1.0399811267852783, 0.40556466579437256, 0.21539118885993958, -2.2056612968444824, -1.3821454048156738, -0.9761359691619873, 0.2517661154270172, 0.824315071105957, -1.2861855030059814, -0.5875123143196106, 0.4553321897983551, -1.6391514539718628, -0.952658474445343, -0.7815382480621338, -0.7780823707580566, 1.3701653480529785, 0.1963454931974411, -1.6314550638198853, 1.3751970529556274, -0.2418382614850998, -0.18272654712200165, 1.4370628595352173, 0.023326965048909187, -0.8225713968276978, -0.8448610901832581, -0.10614798963069916, 0.8461068868637085, 0.8191206455230713, -0.29836589097976685, -1.2220406532287598, 0.8140957355499268, -1.5846765041351318, -0.2615165412425995, 0.7909595966339111, -1.0359442234039307, -0.14325708150863647, -0.9048147797584534, -0.7758225798606873, -0.5011066794395447, 0.6099675893783569, -0.6901649236679077, -0.6691868305206299, 1.451505422592163, 1.3813530206680298, 1.0382506847381592, -0.1011643260717392, -0.013352615758776665, 0.3651044964790344, 0.03344989940524101, 0.08931207656860352, 1.0774458646774292, 2.0793771743774414, 0.3460423946380615, -0.21899957954883575, 0.36457064747810364, -0.14058631658554077, -0.7117235660552979, 0.284437358379364, -0.3660193681716919, 1.2488139867782593, 0.3971916437149048, 0.2915889024734497, -0.5406466126441956, 1.0926709175109863, 1.315454363822937, 0.5563986897468567, -0.5526260733604431, 0.17456242442131042, -2.928770065307617, 0.35114553570747375, 0.8440579175949097, -1.3610762357711792, 0.38369056582450867, -1.2179487943649292, -0.4258391261100769, -0.6177007555961609, 1.5502794981002808, 0.5902612209320068, -0.06969393789768219, 0.024796631187200546, 2.53843355178833, 0.01450528483837843, 0.37405017018318176, -2.458357095718384, 1.2656419277191162, 0.18522192537784576, -0.6567533612251282, -0.29814454913139343, 0.5474624633789062, 0.3527853190898895, -0.9837614893913269, 0.18555371463298798, -0.090118907392025, -0.5635640621185303, 2.0088953971862793, 0.5055131316184998, -1.3682218790054321, 0.11499743908643723, 0.02576841041445732, 0.46136000752449036, -0.6422135829925537, -1.7612881660461426, -1.0086852312088013, 1.9729520082473755, 2.652953624725342, -0.21667221188545227, -2.4930167198181152, -0.9339371919631958, 0.369726300239563, 1.2212746143341064, 0.7183241844177246, 1.124640703201294, 0.6824524402618408, 0.08746112138032913, -0.22691982984542847, -0.9064077734947205, -1.8422040939331055, -0.8748421669006348, -0.37643226981163025, 0.7931452393531799, -1.787496566772461, 1.2409460544586182, -1.5370962619781494, 1.4895660877227783, -0.832904577255249, 0.392973929643631, -0.23791003227233887, -0.03005877695977688, -1.003696322441101, 0.2123408317565918, -0.41769155859947205, 0.5700984597206116, -0.005764513276517391, -0.3238634765148163, -1.6060925722122192], [-0.0032736181747168303, -2.3423848152160645, 0.023723848164081573, -0.08052635937929153, 0.8879179954528809, -1.7249759435653687, 0.8584567308425903, -0.18975099921226501, 0.5540470480918884, -0.35481131076812744, 0.06472478061914444, 1.4265012741088867, -0.24843041598796844, 0.5560517311096191, -0.31469249725341797, 0.110452800989151, 1.1361252069473267, -0.2596243619918823, -0.05810681730508804, 1.1714917421340942, 0.4565693736076355, -0.4765802323818207, -0.07112853974103928, -1.1346501111984253, -0.6090396642684937, -1.3549169301986694, 0.12496112287044525, 0.4291601777076721, -1.861144781112671, 0.9960376024246216, 0.36334067583084106, -0.6784909963607788, -0.827758252620697, -1.0151443481445312, -0.3357745110988617, -0.14448343217372894, 0.8389135003089905, 2.3272883892059326, 0.1710609346628189, 0.6289788484573364, -1.3115565776824951, -0.1408788412809372, 0.6943893432617188, 0.4568113386631012, -1.389424443244934, 1.154420018196106, 0.41088637709617615, -1.4426298141479492, 0.7223774194717407, -1.0388792753219604, -1.4259402751922607, -1.2186923027038574, -0.2492823302745819, 1.3867874145507812, 0.5781665444374084, -0.5582983493804932, 1.3840572834014893, 0.30332666635513306, 0.8935929536819458, -1.5348929166793823, 0.5032838582992554, -0.2916884422302246, -1.3828729391098022, 0.7125111818313599, 0.11041814833879471, 0.16970276832580566, -0.5066311955451965, -1.158699631690979, 1.1568299531936646, -0.05537090450525284, 0.7141335010528564, 0.0024534547701478004, 0.24094323813915253, -0.6855491399765015, 0.27483895421028137, -0.08422525972127914, -0.5909891128540039, -0.26646238565444946, 0.22328847646713257, 0.3945419490337372, -0.047914229333400726, -1.0235278606414795, 0.5070704817771912, -0.515292763710022, -2.339655637741089, -1.3559248447418213, 0.2972753942012787, -0.5813482999801636, 0.386648565530777, 0.4409785270690918, 1.7510168552398682, 0.7277442812919617, -3.678626775741577, -0.8808909058570862, -2.3676669597625732, -0.2131805270910263, -0.26059719920158386, 0.7620887160301208, 0.570528507232666, 0.9375969171524048, -0.5738582015037537, -0.8151793479919434, 0.21321623027324677, -0.4694395363330841, -0.6587417721748352, 1.1880395412445068, -0.3696366846561432, -0.41969969868659973, 0.3556704521179199, -0.2862517237663269, 0.9176706075668335, 1.1463035345077515, -0.7760440111160278, 0.6340619921684265, 0.7528896331787109, 0.42952200770378113, -0.6653929352760315, 2.5114669799804688, 0.31320613622665405, 0.5981564521789551, -0.7083203196525574, 1.9691636562347412, -0.5255618095397949, -1.0755913257598877, -1.1854021549224854, -0.2887168526649475, -1.529788613319397, -0.3425174951553345, -0.06779598444700241, 0.9241818785667419, 1.025479793548584, 0.9519346356391907, 1.0064858198165894, 2.243260622024536, 0.22369131445884705, 1.2677491903305054, 0.8074643611907959, 0.4458445608615875, -1.6725356578826904, -0.07869034260511398, 1.5871347188949585, -1.0039387941360474, -0.6272815465927124, -0.06012296676635742, 0.4491785764694214, 0.1818893998861313, -0.15769676864147186, -1.5120466947555542, 0.8136450052261353, -0.4390682876110077, -0.5076896548271179, -0.6078418493270874, 0.47740817070007324, -2.572263717651367, -1.5501517057418823, 1.13041353225708, 0.46077561378479004, -1.072932481765747, 1.8051443099975586, -0.8340243101119995, -0.03925912082195282, 0.7114179730415344, -0.8880069255828857, 0.5593824982643127, 0.44814130663871765, -0.8587943315505981, -1.3226723670959473, 1.4041138887405396, -0.013114572502672672, 1.9235124588012695, 0.7473193407058716, 0.37041473388671875, 0.03130422160029411, -0.812039315700531, 0.2521759271621704, -0.6040739417076111, 1.4464824199676514, 1.1904345750808716, -1.5775386095046997, -0.9320762157440186, -0.9436696767807007, 0.7890371680259705, 0.47839558124542236, 0.5184858441352844, -0.2525534927845001, 0.7381478548049927, -1.5738509893417358, 0.7496842741966248, 0.22762297093868256, 0.12380021810531616, -0.38839033246040344, -2.8835768699645996, 0.9883747696876526, -1.0854771137237549, 0.363945871591568, -0.8913785219192505, 0.5878074169158936, -0.09703034162521362, 0.7692486047744751, -0.23046693205833435, 0.5488778352737427, 0.3333892226219177, -2.297781467437744, 1.102964997291565, 0.32911479473114014, -0.1799800843000412, -0.3072434961795807, 0.3998008668422699, -1.1350457668304443, -0.3858165442943573, -0.2386532425880432, 1.8775629997253418, -1.034597396850586, -1.7719366550445557, -0.8781561851501465, -0.6831550598144531, -0.556121289730072, -0.2517212927341461, 0.5355347990989685, 0.29925206303596497, 0.01271966565400362, 2.525736093521118, 1.1409536600112915, 0.8126669526100159, -0.016087999567389488, -0.8221309185028076, 0.32670584321022034, -1.9557445049285889, -0.7204806804656982, -0.6336483955383301, -1.2413747310638428, 1.935900330543518, 1.2619996070861816, -0.36656010150909424, -1.166693091392517, 0.7919752597808838, 0.10284098237752914, -0.16567538678646088, 0.772641122341156, -0.937160074710846, 0.285334974527359, 2.339444637298584, 0.757265031337738, 0.5165601968765259, 1.9918339252471924, 1.2027647495269775, 1.747139811515808, -0.500984251499176, -0.6188702583312988, 0.12199399620294571, -0.7291486859321594, -1.1341623067855835, 1.6800329685211182, -0.30204251408576965, -1.5617426633834839, 0.27338600158691406, -0.023028893396258354, -0.08322176337242126, 0.09065693616867065, 0.3933677673339844, -1.0547488927841187, -0.26759281754493713, 0.17298996448516846, 0.9309361577033997, 0.07103394716978073, -1.1010606288909912, 0.6769946813583374, -1.1653825044631958, 0.022657448425889015, 0.48517942428588867, -0.23994572460651398, 0.7034085988998413, -0.21884623169898987, -0.6406539678573608, -0.2652207612991333, 0.2194671332836151, 0.5551007986068726, -1.671555519104004, 0.31992658972740173, -0.4306514859199524, 1.0183814764022827, -1.9576042890548706, 0.33996647596359253, -0.6770851016044617, -0.5231837034225464, -1.176468849182129, 1.1995552778244019, -1.4764189720153809, 0.6905248165130615, -1.5703572034835815, 0.5275385975837708, -1.5241541862487793, -0.29014769196510315, 1.5569506883621216, 0.6799969673156738, -1.7852011919021606, 0.0004937941557727754, -3.0587196350097656, 0.26455608010292053, -2.2089524269104004], [0.4292198419570923, -0.38092654943466187, 0.6650213003158569, -0.7237395644187927, 0.8382972478866577, 0.27002349495887756, 1.690144419670105, -1.031272292137146, -0.4109386205673218, -0.9877899885177612, -0.3891685903072357, 0.8637123703956604, -0.1565411537885666, -0.6058910489082336, -0.3282209038734436, -0.2791281044483185, 0.011581258848309517, 1.5509910583496094, -0.17179521918296814, 2.079782247543335, 0.6232550740242004, 0.5378130674362183, -1.1747901439666748, -0.7164985537528992, -0.5282799601554871, 0.9009802341461182, -1.3524483442306519, -0.6659271717071533, -1.0045322179794312, 1.0674092769622803, -1.6488455533981323, 0.5527706146240234, 0.4173702001571655, 0.24083559215068817, 1.5857632160186768, 0.564803957939148, -1.2900364398956299, 0.7487949728965759, -0.4989250600337982, 0.6943267583847046, -0.5970335602760315, -0.3315180540084839, -1.5227547883987427, -0.43734651803970337, -1.4816153049468994, 0.13066981732845306, 0.20353157818317413, 1.8586149215698242, -1.1061947345733643, 0.6236889958381653, -0.022282762452960014, -0.8171308636665344, 2.1587073802948, -1.6822744607925415, -0.5874804854393005, 0.8893799185752869, 0.20227015018463135, 0.24231404066085815, 0.7317284941673279, 1.1022347211837769, -2.0699985027313232, 0.3496367335319519, 0.021571071818470955, 0.831122100353241, 1.644771933555603, 0.3432160019874573, -0.09437556564807892, 0.5549439787864685, -1.4942106008529663, 0.9594201445579529, 0.5594854950904846, -0.936327338218689, -0.6978177428245544, -0.2995890974998474, 0.7535620331764221, 2.269550085067749, 2.731674909591675, -0.9007960557937622, -0.5734132528305054, -0.37052401900291443, 1.1751022338867188, 0.12452229857444763, -1.6990349292755127, 0.1727903038263321, -1.5048242807388306, 0.8149487972259521, -0.009369255974888802, 1.185550332069397, 0.451750248670578, -0.7559455633163452, -1.1583664417266846, -1.3940153121948242, -0.3004972040653229, 1.9747501611709595, 1.7597583532333374, -0.9507054686546326, -1.1260318756103516, 1.7824366092681885, -0.8249141573905945, 0.08079802244901657, 0.08083513379096985, -0.5162918567657471, 1.0118002891540527, 0.8834792971611023, 0.3363484740257263, -1.5140856504440308, -1.453061819076538, 0.32674288749694824, -0.5495915412902832, 0.7831010222434998, -0.5262419581413269, -1.0329755544662476, -0.8336142301559448, 3.0640482902526855, -0.606979250907898, -0.5381540656089783, -0.983289361000061, -0.21547077596187592, 0.18227162957191467, 1.6545839309692383, 1.072041630744934, -0.501802921295166, 0.005312792025506496, 0.15746057033538818, 0.17753875255584717, 1.107808232307434, 1.3363252878189087, 1.5303006172180176, 0.8916653990745544, 0.20118123292922974, 0.18037624657154083, -0.19225211441516876, -0.012867383658885956, -0.14341354370117188, 0.18429693579673767, -0.30723652243614197, 0.6797906160354614, -1.9488681554794312, 0.8109579682350159, 1.2742786407470703, 0.37421220541000366, -0.6790114641189575, -1.7149296998977661, -0.16503214836120605, 1.192623496055603, 0.09681820124387741, -0.6247556209564209, 0.8055386543273926, 0.3116368055343628, -0.6851646900177002, -0.7983474135398865, 0.8983595967292786, 1.2769237756729126, 0.22924719750881195, 1.4165465831756592, 0.1748332679271698, 0.7260031700134277, -1.2314764261245728, -0.5741836428642273, -1.5649946928024292, -0.5668961405754089, -1.1852912902832031, 1.677056074142456, 0.11104439944028854, 1.2178760766983032, -0.26852118968963623, 0.8993208408355713, -1.3682225942611694, 0.7524467706680298, -0.28241556882858276, 0.7867327928543091, 0.3713933527469635, -2.511601209640503, 1.9840219020843506, -1.1416008472442627, -1.420613408088684, -1.8516474962234497, -0.42678940296173096, -1.5230331420898438, -0.9609089493751526, -0.0017094918293878436, -0.6635841131210327, -0.6992473006248474, -0.37988951802253723, -2.0704798698425293, 0.22563552856445312, 1.5489733219146729, -1.5877699851989746, 0.2851756811141968, 0.2859029769897461, 0.9313081502914429, -1.6226897239685059, 1.0506396293640137, 0.4646661579608917, -1.9688479900360107, -1.1240737438201904, -0.24102169275283813, -0.025778688490390778, -0.473145455121994, -1.5613192319869995, 0.4387170672416687, 1.4636815786361694, -1.3759251832962036, -2.13881778717041, 0.5085686445236206, 0.7941079139709473, -0.4996475279331207, 0.7887288331985474, 0.3808305859565735, 1.6158745288848877, 1.0694102048873901, -0.15861402451992035, 0.40907222032546997, -1.0458338260650635, 0.7127491235733032, -0.47397926449775696, 1.184334635734558, -0.3684845566749573, -1.4474132061004639, 1.0536918640136719, 0.34677746891975403, 0.7816737294197083, 1.4435402154922485, -0.8290126323699951, -0.6780103445053101, 0.07307565957307816, -0.9841950535774231, -1.9576959609985352, 0.28172194957733154, 0.9484179019927979, -0.6140354871749878, 0.9745141863822937, 0.3576439321041107, 0.6251272559165955, -0.3275506794452667, -0.01084564346820116, 1.195274829864502, 1.5907894372940063, 1.5343719720840454, -0.46790871024131775, 0.046489596366882324, 0.6480207443237305, 1.2076044082641602, -0.09142962843179703, 0.44118234515190125, -1.7249531745910645, -0.2906968891620636, 0.794920802116394, 0.553571879863739, 1.9090372323989868, -1.105028510093689, -0.8793430924415588, -1.3582637310028076, 1.1913325786590576, -0.4992973804473877, -0.7488034963607788, -0.48159727454185486, 0.0479649156332016, 0.11840854585170746, 0.6361997127532959, -1.397039532661438, -0.023225698620080948, -0.04683874920010567, -2.220477342605591, -0.9340409636497498, 0.5131092071533203, 0.4312041401863098, -0.7358383536338806, -0.40798211097717285, 0.37368321418762207, -2.154744863510132, 0.7594385147094727, 1.7095918655395508, -0.2770882248878479, 0.2760637402534485, 1.1970555782318115, -0.46620872616767883, 0.6824826002120972, 0.17347925901412964, 0.6649907827377319, -0.2046087682247162, 0.7947351336479187, 0.46573907136917114, 0.17044520378112793, -0.41329696774482727, 1.224247932434082, 0.02327299304306507, 0.992813229560852, -1.3744163513183594, -0.07775852829217911, 0.2436683028936386, 1.768176794052124, 0.8996677398681641, 0.38275256752967834, -0.47198066115379333, -0.939751923084259, -0.3416841924190521, 0.4870973527431488, -0.546997606754303, -1.9354114532470703], [-1.1831895112991333, 0.13754476606845856, -0.6771450042724609, 0.6395711898803711, -1.1668568849563599, -0.644646167755127, 1.5855001211166382, -0.4172302484512329, 1.7613282203674316, -0.7207856178283691, 2.116279125213623, 1.4959723949432373, -0.7564013600349426, 0.4134403467178345, -1.6971880197525024, -0.847194254398346, 0.4470211863517761, 0.4827335774898529, -0.17747260630130768, 1.0684785842895508, 0.3258805572986603, 1.507290005683899, -0.43705710768699646, 0.3263295590877533, -0.687390148639679, 2.7620086669921875, 0.8215804100036621, 0.1744844764471054, 0.9107087254524231, 1.2011138200759888, 0.6574773192405701, -1.891748309135437, -0.9206157922744751, 1.0120859146118164, 0.12474978715181351, -0.33088499307632446, -0.5878517031669617, -0.1794634610414505, 0.7624014616012573, -1.1407244205474854, -1.3850138187408447, -0.49099358916282654, -0.13953396677970886, -0.4667697250843048, 0.9864100217819214, -0.8982951641082764, -0.37702494859695435, 0.23208993673324585, -0.8040871620178223, 1.2345292568206787, -1.1860238313674927, -0.4337935745716095, 0.15673889219760895, 0.9882592558860779, -1.9383128881454468, -0.36956098675727844, -0.845282793045044, -0.5742272734642029, -0.30876269936561584, 0.5860258340835571, -0.4766862690448761, 0.23538802564144135, 1.7424408197402954, 1.2761362791061401, -0.3568758964538574, 0.7726261019706726, 0.12028554081916809, 1.4203141927719116, -0.0376785472035408, 0.04369932413101196, -0.7175458073616028, -0.8241099119186401, 1.249014139175415, 2.37487530708313, -0.01652146317064762, 0.14742060005664825, 0.819621741771698, 1.633730173110962, 0.16273875534534454, 0.32065629959106445, 0.350559264421463, -0.0524556040763855, -1.4317668676376343, -0.5322107076644897, -0.08093511313199997, 1.035793423652649, -0.2540263235569, 1.2797062397003174, 0.5302005410194397, -0.6182062029838562, 0.7522777915000916, 0.13557419180870056, -0.2741570472717285, 0.5905079245567322, 0.35864073038101196, -0.6513127684593201, 0.3310413062572479, 0.49468493461608887, -0.8166183233261108, 0.0025792717933654785, -0.7199265956878662, -0.8320128917694092, -0.5498971343040466, -0.31921693682670593, -0.9114824533462524, -0.629909098148346, 0.08132950216531754, 0.19522693753242493, 0.14924746751785278, -0.7179086208343506, 0.3981163203716278, 1.0679588317871094, -1.5294225215911865, 1.2024575471878052, -0.22987709939479828, 0.7761753797531128, -0.9049001336097717, 2.5650618076324463, 0.140410915017128, 0.8308059573173523, 0.8265897035598755, -0.17013095319271088, 2.0552499294281006, 0.9813132882118225, 0.2917553186416626, 2.1178908348083496, 0.29046234488487244, -1.2533968687057495, 1.3224458694458008, 0.18203429877758026, 0.19887475669384003, 0.6110673546791077, 0.8779538869857788, 0.4260220229625702, -0.04710821062326431, -0.12322887778282166, -1.1029517650604248, 0.08435092866420746, 0.47732841968536377, -0.707808256149292, 0.8098239302635193, 0.23545902967453003, 1.0857282876968384, 0.2728673815727234, 0.38919928669929504, -0.397634893655777, 2.2305550575256348, 0.09972091764211655, -0.6721295714378357, -1.49282968044281, 0.6383809447288513, 0.9658546447753906, -1.3680105209350586, -1.2364391088485718, -0.5857328176498413, -0.6117905378341675, -0.7055023312568665, 0.11508958786725998, -0.30205637216567993, 1.3166933059692383, -0.29474887251853943, -1.1607139110565186, 0.06464945524930954, -1.2846554517745972, 1.0553003549575806, -0.08113428205251694, 0.3115578889846802, -1.1713347434997559, -0.7344979047775269, 1.4057496786117554, 2.0030593872070312, 0.7978310585021973, -1.2736873626708984, 0.9576990604400635, -0.7711796760559082, 1.1604853868484497, -1.5768489837646484, 0.9472683668136597, -1.5319572687149048, -0.039397355169057846, -0.5212151408195496, 0.901627779006958, -0.5263317823410034, -0.4732078015804291, -1.6138346195220947, 0.27906206250190735, -0.5483778119087219, -0.10404250025749207, 1.3417577743530273, 0.5078028440475464, 1.5896620750427246, -1.6315250396728516, 0.511136531829834, 0.7735824584960938, -0.8930290341377258, -0.5638577938079834, -0.15351040661334991, 0.751746654510498, 1.7174385786056519, -0.6865884065628052, 1.3025058507919312, 0.39486971497535706, 1.7425919771194458, -0.3614386022090912, 0.6368296146392822, 1.231693148612976, 0.6765667796134949, 0.008799629285931587, -0.8391253352165222, 1.5805468559265137, 0.8725740909576416, -0.8974515199661255, 0.26201000809669495, -1.673116683959961, 1.2122607231140137, 1.4800083637237549, -1.9891490936279297, 0.36485809087753296, 0.2243404984474182, 0.4236923158168793, 2.1654868125915527, -0.5352563261985779, -0.5123922824859619, -0.6748578548431396, 0.6460710763931274, -0.9890856742858887, -2.172945261001587, 0.15435853600502014, 2.1932616233825684, -0.9280948042869568, -0.4376929998397827, -0.23511135578155518, -0.7946918606758118, -1.7866473197937012, 1.214098334312439, 0.4150579869747162, 0.3944449722766876, -0.9929917454719543, -0.45758160948753357, 1.1623024940490723, 1.3610925674438477, 0.35711389780044556, -2.0240910053253174, -0.044044822454452515, 0.288606196641922, -0.06699171662330627, -0.5571460723876953, -0.6615786552429199, 0.828290581703186, -0.005729552358388901, 0.05303577333688736, -0.10551882535219193, -1.280835509300232, -1.9025390148162842, 0.9246530532836914, 1.5270992517471313, 3.1305277347564697, -0.4581165909767151, -0.28696686029434204, 1.2027699947357178, -0.04205802455544472, -0.601861834526062, 0.6310965418815613, 0.4875853359699249, -1.0102076530456543, -1.0968708992004395, -1.36484694480896, -2.482649803161621, -1.0497715473175049, 0.6074047684669495, -0.5835953950881958, -1.6014317274093628, -0.7652906775474548, 0.9676340222358704, -0.2861168384552002, -1.046707034111023, -0.07833133637905121, -0.9698466658592224, -0.3682899475097656, 0.018690241500735283, -2.8860342502593994, -0.4456269443035126, -0.785620927810669, -1.3570263385772705, -1.5824147462844849, -0.02656175196170807, 0.8682524561882019, -0.321027547121048, 0.4115578532218933, 2.4823694229125977, 0.6064878106117249, 0.06968297809362411, -1.9044036865234375, -0.034612514078617096, 0.2592126429080963, 0.6281262040138245, -1.3678500652313232, -0.6018540263175964, 0.31101253628730774, -0.09225915372371674], [-0.3735489845275879, 0.015216346830129623, 1.0296120643615723, -1.0399104356765747, -0.7029631733894348, -0.632848858833313, -0.4647466838359833, -0.5369998812675476, 0.73532634973526, -0.2659291923046112, -0.08872600644826889, -0.8760994076728821, -1.1840208768844604, -0.9488844871520996, -0.5109367966651917, 0.42816153168678284, 0.10928858071565628, -0.020164674147963524, -1.0760705471038818, 0.40667104721069336, -1.3532477617263794, -0.4106804430484772, 0.9940956830978394, 0.6191776990890503, -0.34199875593185425, -1.044775366783142, 0.7389501929283142, -1.489445447921753, -0.016183357685804367, 1.0441155433654785, 0.6117901802062988, 0.47070419788360596, 0.4256269931793213, 0.27713388204574585, 1.272114872932434, -0.7782129049301147, -1.5538718700408936, -1.1553003787994385, -0.9242016077041626, -0.02361552231013775, 0.7216994166374207, -0.012427918612957, -0.4332234561443329, -0.403628945350647, -2.2771060466766357, -1.341092824935913, -0.5138042569160461, -0.3717062473297119, -0.5929148197174072, -0.5725383162498474, 0.7595463991165161, -0.5373963713645935, -0.6097280979156494, -1.4565340280532837, 0.4963442385196686, -0.6083073616027832, 0.24130016565322876, 0.8609235286712646, -2.061332941055298, 0.30490368604660034, -0.5722832679748535, -1.6500189304351807, 0.45265457034111023, 1.0051718950271606, -0.14677086472511292, -0.6508864760398865, -1.0867490768432617, -0.7398656606674194, -0.502325177192688, 0.3572714924812317, -0.7064923048019409, -0.2687511742115021, -0.1665801703929901, -0.717817485332489, -0.27067139744758606, 0.8390399217605591, -1.9929955005645752, -0.0957115888595581, 1.2219905853271484, 1.8827859163284302, 0.09194642305374146, 0.6390318274497986, -1.2887874841690063, 0.5012468695640564, -0.07749587297439575, -0.6230360269546509, 1.5684505701065063, -0.8484800457954407, 0.4988609254360199, 0.12636452913284302, -1.3895232677459717, -1.6453279256820679, 1.2621872425079346, 0.15451301634311676, -2.295935869216919, 0.35650771856307983, -0.1476067751646042, 1.8867511749267578, -0.049541257321834564, -0.4458518922328949, 0.3673289120197296, -0.5221341252326965, -0.2576371431350708, -0.33951374888420105, 0.5000248551368713, -0.21930311620235443, -0.3422897756099701, -0.11938902735710144, -0.38504859805107117, 0.38722559809684753, 0.7274322509765625, 0.3447689116001129, -0.2127624750137329, -2.804978132247925, 0.7012112736701965, -1.7640773057937622, -0.04892786219716072, -1.0385538339614868, -0.8009041547775269, -0.06277074664831161, 1.1026207208633423, 0.18246401846408844, -1.3458137512207031, -1.123684048652649, 0.8635172843933105, 0.37000414729118347, -0.41585931181907654, 1.0490444898605347, 0.048030950129032135, 0.01730026863515377, -0.4045215845108032, -1.9212331771850586, 1.1471514701843262, 0.2991262674331665, -0.15309131145477295, -0.02463848888874054, -0.05865208059549332, -0.6343912482261658, -0.5579099655151367, 0.4307502210140228, -0.2065148502588272, 0.08349037915468216, 0.008961433544754982, -0.43298104405403137, -1.1398810148239136, 1.4373549222946167, -1.3303825855255127, -0.8115243911743164, -1.3696244955062866, 0.2922646701335907, 1.4498703479766846, 1.5754557847976685, -1.2703417539596558, -0.48883897066116333, -0.8276026844978333, -0.047088176012039185, -0.7819859385490417, -1.2324787378311157, 0.8452898859977722, -1.510055422782898, 0.9406841397285461, -0.7601470947265625, 0.17992663383483887, -0.11115870624780655, 1.1256401538848877, 0.44025206565856934, -0.6230040192604065, -1.2174016237258911, 1.463364839553833, 0.6072016954421997, 0.5468044877052307, -0.9863866567611694, 0.3327382802963257, -0.33969026803970337, -0.542320728302002, 0.6628072261810303, 1.0459022521972656, -1.3292165994644165, 0.7394420504570007, -0.36843180656433105, 0.2636033892631531, -0.11262361705303192, 1.1711530685424805, -0.42136451601982117, 2.232510805130005, 1.129646897315979, -0.0314302034676075, -1.2438042163848877, 1.0097061395645142, 1.1098830699920654, -2.471906900405884, 0.47063764929771423, 0.3371458053588867, 1.7249772548675537, 1.6438864469528198, -1.3029426336288452, 0.21632499992847443, 0.1550244390964508, 1.0165650844573975, -2.0061562061309814, -0.37383949756622314, -0.2976841926574707, -2.6575927734375, 0.5248538851737976, -0.03707895055413246, 0.4379354417324066, 0.6775729656219482, -0.45198363065719604, -0.8857209086418152, -0.18868210911750793, 0.19035619497299194, -0.436689168214798, -0.7423117756843567, -1.5316280126571655, -0.6497320532798767, 1.559227705001831, 1.6215909719467163, 0.5310773849487305, 0.23106536269187927, -0.9085186123847961, -1.0249460935592651, -0.8453614115715027, -1.1931796073913574, -0.4996059834957123, 0.11781081557273865, -1.0396136045455933, 0.3758617341518402, 0.09809599071741104, 0.211440771818161, 1.1881680488586426, 0.5532371401786804, -1.5057669878005981, 0.07291059195995331, -0.6340408325195312, -0.367085725069046, -0.7298763990402222, -1.7982221841812134, 1.035431981086731, -0.7241026163101196, 1.9862204790115356, 1.4883126020431519, -0.8685718774795532, -1.8578145503997803, 0.45208391547203064, -0.0956154316663742, 1.9711945056915283, 0.3466942310333252, 0.18901744484901428, -0.8964731097221375, 0.280019074678421, -1.357045292854309, -1.0762120485305786, 0.5557715892791748, -1.1331477165222168, 0.8218944072723389, 1.418013095855713, 0.029407232999801636, -0.46752050518989563, 1.1491469144821167, 0.7153054475784302, -0.24448458850383759, 0.684274435043335, 0.83106929063797, -0.8176936507225037, -1.8635770082473755, -0.10587691515684128, 1.71233069896698, 1.9083901643753052, -0.8009986877441406, 1.100433111190796, 1.5867855548858643, -0.5698921084403992, -0.9809578657150269, 0.09634485095739365, 1.0428811311721802, -0.70954430103302, 2.2085704803466797, -0.46127575635910034, -1.1042215824127197, 0.8103824257850647, -0.008511107414960861, 1.196398138999939, -1.164763331413269, -0.49271294474601746, -0.05363784730434418, -0.3565205931663513, -0.15832006931304932, -0.7124928832054138, 0.077630914747715, 0.9492979049682617, 0.4650122821331024, -1.2318532466888428, 0.03088623844087124, -0.046027421951293945, 0.13861453533172607, 0.49190935492515564, 0.6345593929290771, -0.5033612251281738, -0.9727732539176941, 0.5141489505767822], [-0.41261863708496094, -0.010071687400341034, -0.15402495861053467, -1.2685390710830688, -0.10336343944072723, -1.3144683837890625, -0.3069271743297577, -0.012038902379572392, -0.6854205131530762, -1.0972700119018555, -1.5668216943740845, 0.2420257180929184, -0.9509410858154297, 2.0092790126800537, -1.869559645652771, -0.2301371842622757, 0.6551317572593689, -1.3131530284881592, -0.5997835993766785, -1.3511847257614136, -0.5146631598472595, -0.44792017340660095, 1.602052927017212, -1.0797765254974365, 0.4825901389122009, 1.8401576280593872, -0.5694467425346375, -0.9721371531486511, 2.0127906799316406, -0.40184012055397034, 2.3904900550842285, 0.14296530187129974, 0.2362741231918335, 0.9213754534721375, 0.8959159851074219, -1.3093016147613525, -0.5725783705711365, -0.9075754284858704, 0.48042869567871094, -1.3286484479904175, 1.380448341369629, 0.5687965154647827, 0.20665381848812103, -0.6188362240791321, 0.5766984224319458, -0.8728491067886353, 2.3168258666992188, 0.8606046438217163, -1.5624675750732422, 0.5899501442909241, 1.260678768157959, 0.14850755035877228, 2.346370220184326, -0.6602331399917603, 0.9347837567329407, -0.6676376461982727, 0.2085018903017044, 0.8552208542823792, -0.3599690794944763, -0.8779547214508057, 0.8805140852928162, 0.9531556963920593, 0.13015176355838776, 1.2286207675933838, -1.7015290260314941, -1.6826417446136475, 1.1601622104644775, -0.3573899269104004, 0.188268780708313, 0.7175699472427368, -0.8622449040412903, -1.6949251890182495, 1.0547221899032593, 0.5672406554222107, 0.3918532133102417, -0.14780938625335693, -0.40462103486061096, -0.5981718301773071, -0.15558947622776031, 1.6999620199203491, -0.31290435791015625, 0.025368940085172653, -0.45116719603538513, 1.595860481262207, 1.3459868431091309, -0.9804449677467346, -0.172354057431221, -1.2930089235305786, 0.16028675436973572, -0.8323203325271606, 0.33958399295806885, 1.1947194337844849, -0.8330290913581848, -0.9363382458686829, 0.054360873997211456, 0.355353444814682, 0.4928927719593048, 0.5935673117637634, 0.5761389136314392, -0.3728380501270294, 0.15299230813980103, 0.2952430248260498, 0.5008859038352966, 0.6757673621177673, -0.9239662289619446, -1.0692968368530273, 0.023718204349279404, -0.3780301511287689, -0.15540307760238647, 0.1735941767692566, -0.6988304257392883, 0.9674403667449951, 1.0391005277633667, -2.721339464187622, -0.4679702818393707, 2.3693723678588867, -1.3761221170425415, 1.5410642623901367, -1.6507627964019775, -0.5868965983390808, -0.4737277626991272, 0.9618741869926453, 0.14211802184581757, -1.0683501958847046, 0.36556410789489746, 0.1890164017677307, -2.1502530574798584, -0.6311749219894409, -0.08577834069728851, -1.5654631853103638, -0.5615450739860535, -1.0176185369491577, -1.2912793159484863, 0.7050189971923828, 1.5140085220336914, 0.18445651233196259, -0.9033466577529907, 0.5528676509857178, 1.115584135055542, 0.2515212595462799, 1.2889336347579956, 1.7759816646575928, -0.9748033881187439, 0.8419481515884399, 1.3035699129104614, 1.4086921215057373, 0.38584986329078674, 0.8337264657020569, 1.532479166984558, -0.5966856479644775, 0.5550758838653564, 0.5596237182617188, 0.2292703539133072, -1.31344735622406, -0.6303978562355042, -0.48927757143974304, -0.4032406210899353, 0.3531825840473175, -1.3108665943145752, -1.675657033920288, -1.2911921739578247, -0.47428861260414124, -0.8045098185539246, 0.744383692741394, 0.6223012208938599, -1.1646754741668701, 0.8532876372337341, -0.510295033454895, -1.535355806350708, -0.21727007627487183, 0.034176938235759735, 0.4539870023727417, -0.6416765451431274, -1.2522578239440918, -0.9482762217521667, 1.490746021270752, -0.31681445240974426, -0.4772350788116455, -0.00987325981259346, 0.8039028644561768, 0.5252727270126343, 0.6413890719413757, -1.0201658010482788, -0.43113061785697937, 1.3647180795669556, -1.1250122785568237, -0.3994976580142975, -0.3554908335208893, 1.0321224927902222, 2.284620523452759, 1.5841176509857178, -0.7374683022499084, 0.5126087665557861, 0.4506729245185852, -0.9381365776062012, -0.24945855140686035, 0.012292377650737762, 0.9499358534812927, 1.8012148141860962, -0.7851381301879883, -0.12439710646867752, 0.1672629863023758, -0.4860110282897949, -1.9187861680984497, -1.8485603332519531, 0.20669618248939514, -1.0803319215774536, 1.014910101890564, 1.1683651208877563, 0.34273388981819153, 0.3547171950340271, -0.04057694226503372, 0.5816471576690674, -0.38247430324554443, -0.4575613737106323, 2.604212760925293, 0.23532018065452576, -0.8603216409683228, 0.16680285334587097, -1.4818774461746216, -0.23306287825107574, -0.27903884649276733, -0.6151820421218872, 1.8641873598098755, -0.11682135611772537, -0.10758908838033676, 0.22365808486938477, -1.6826432943344116, -1.530204176902771, 0.13041047751903534, 0.7331104874610901, 1.610178828239441, 0.5891452431678772, -0.37702295184135437, 0.05624157935380936, -1.4509364366531372, -0.24735188484191895, -1.0407171249389648, 0.04754234850406647, 0.07243946939706802, -0.5348264575004578, 0.9721601605415344, 0.685080885887146, 2.3446402549743652, -1.2633053064346313, -0.7974016666412354, 1.7035727500915527, 0.9497296214103699, -1.5001633167266846, -0.9703041315078735, -1.1629749536514282, -0.7644464373588562, 0.570796549320221, 0.06907568126916885, 1.3003615140914917, -0.7825299501419067, 0.7510529160499573, -0.8169943690299988, 0.623385488986969, 1.8366460800170898, -0.4669609069824219, 1.9703885316848755, -0.6409355998039246, -1.0546764135360718, -0.7699140310287476, -1.4073561429977417, 0.19998660683631897, -0.6444911956787109, -0.6457483768463135, -0.05553112179040909, 0.12440817058086395, -0.06752758473157883, -1.742339015007019, -0.27673354744911194, -0.3150189518928528, -0.7275158762931824, -1.5531890392303467, 1.0819216966629028, -0.6805474162101746, 0.1469317227602005, 0.47674044966697693, -0.9552943110466003, -1.4194196462631226, -0.2861669361591339, -0.304370641708374, -0.09145267307758331, -0.11839884519577026, 0.4866577386856079, -0.5522240996360779, -1.1552308797836304, -0.23349037766456604, 0.0013570486335083842, 0.09960610419511795, -0.453235000371933, -0.8017549514770508, 0.4878813922405243, -1.169693946838379, -0.7900602221488953, 0.8701192736625671, 0.29879283905029297], [-0.7987866997718811, -0.21633802354335785, 0.19495831429958344, -0.37836915254592896, -0.6304348707199097, -0.8090269565582275, -0.72535240650177, -0.2500740885734558, 0.05754910036921501, 0.7209218144416809, 0.27345114946365356, -0.24693314731121063, 2.8425545692443848, 1.3269824981689453, -0.10269799083471298, -1.0817053318023682, -1.3787592649459839, 1.2665506601333618, 1.4625391960144043, 1.770469307899475, 0.007435000967234373, 1.1482447385787964, -0.2936471700668335, 1.6341127157211304, 1.0772348642349243, -1.8310126066207886, 0.5218149423599243, 1.414995551109314, -1.1802833080291748, -1.047295331954956, 0.3897611200809479, -1.3783820867538452, 1.7766647338867188, 1.3872628211975098, 0.601534903049469, -1.7497880458831787, 0.6983282566070557, -1.2658300399780273, -1.1172116994857788, -0.7235516309738159, 0.47991397976875305, 0.0913577601313591, 0.9381160736083984, 1.7389765977859497, -1.1016546487808228, 0.035806890577077866, 1.2193019390106201, 0.5967705249786377, 0.9155547618865967, -1.2372643947601318, 0.8421862125396729, -0.1900223195552826, 0.4022156894207001, 0.3483230173587799, 0.02740791067481041, 1.7606357336044312, -1.166043996810913, -0.47398731112480164, -0.6153954267501831, -1.50892174243927, 0.4489034116268158, 0.03211892396211624, 1.2099286317825317, -1.0669641494750977, 0.691132128238678, -1.2664448022842407, -2.843769073486328, -0.14657637476921082, 1.770405888557434, -0.46570804715156555, -0.3287692666053772, 0.2530071437358856, 0.650968611240387, -1.1807585954666138, -0.07770741730928421, -0.19107480347156525, 0.6846176385879517, -0.2618058919906616, 0.4515402615070343, 0.7433054447174072, -0.004514778032898903, 0.23687992990016937, -0.3243301808834076, -0.6936789751052856, 0.41867712140083313, 0.5483328104019165, -0.6580705642700195, -0.4090617299079895, -1.0889899730682373, 1.1274359226226807, -0.8343594670295715, 0.19026778638362885, 0.33245572447776794, 2.077216625213623, 0.9858380556106567, -0.12721258401870728, 1.4085485935211182, -0.9702765941619873, -0.5189021229743958, 0.6851783394813538, 0.3823874890804291, -0.05376720428466797, -1.0926074981689453, 0.03550056740641594, 1.3158413171768188, -1.0915911197662354, 2.1487441062927246, 0.784763753414154, -1.136574625968933, -0.26689642667770386, -0.7223770022392273, -0.3925483524799347, 1.3971129655838013, 0.7958187460899353, -1.0546472072601318, 1.1206252574920654, -1.011006236076355, 0.38702335953712463, -0.6644152402877808, 0.4367009997367859, -1.6002010107040405, 0.41441190242767334, -0.1816636174917221, -0.2213200479745865, 1.4509061574935913, 0.4537815749645233, -0.023059207946062088, 0.06692704558372498, 0.03579581901431084, -1.7052394151687622, -0.1856347769498825, -2.0913896560668945, 2.0445547103881836, -0.11961186677217484, 0.3250405490398407, 0.8053191304206848, -1.6694481372833252, 0.1381574124097824, -0.6333960294723511, 1.7147717475891113, -0.10892488807439804, 1.0142192840576172, 0.3444753885269165, 1.1774741411209106, 0.7537446618080139, 0.5007703900337219, -0.16470523178577423, 0.8651108145713806, -0.031868599355220795, 1.2591761350631714, 0.5371088981628418, 0.9917367696762085, -0.06872421503067017, 0.8253242373466492, -1.076206922531128, -1.6200518608093262, 0.8716931939125061, 1.5989549160003662, 1.5396991968154907, 0.33368605375289917, -1.2269377708435059, -0.6603665947914124, 1.5984798669815063, -0.04836331680417061, -0.41859006881713867, -0.9652706384658813, 0.2743823230266571, -0.557624340057373, -1.68125319480896, 0.3666217625141144, -0.0710851326584816, -0.531801164150238, 0.2231730818748474, -1.3493835926055908, 0.09025657176971436, -0.9231249690055847, 0.28733590245246887, -0.15295660495758057, -1.3159050941467285, -0.20735757052898407, 0.47825565934181213, -0.38392022252082825, 0.6804462671279907, -0.9893813729286194, -0.009291657246649265, 1.2877007722854614, 0.17637254297733307, 0.5925905108451843, 1.7626749277114868, 0.06578614562749863, -1.3828670978546143, -0.2846573293209076, -0.753656268119812, -0.20482906699180603, -0.4830002188682556, -0.43207472562789917, -0.17771820724010468, 1.5969948768615723, 0.17500674724578857, -0.9535583257675171, 1.307572364807129, 1.5247722864151, 0.05238349363207817, -0.605567455291748, 1.6232149600982666, 0.6120725870132446, 0.9669826626777649, 0.13973134756088257, -0.9993562698364258, 1.7431076765060425, -0.04514813423156738, 0.46695685386657715, -1.7483177185058594, -0.45426711440086365, -0.3480108380317688, 0.6347865462303162, 0.8198089599609375, -0.6532905101776123, -2.1469359397888184, 0.31517037749290466, -0.309857577085495, 1.065750241279602, -0.7817955017089844, 1.076494812965393, -1.1620427370071411, -0.8182796239852905, -0.19185957312583923, -0.03520798683166504, 0.9164595007896423, 0.1517961323261261, -1.8353537321090698, 1.8965266942977905, -0.7611806988716125, 0.678318440914154, 0.365628182888031, -0.09865407645702362, 1.0013607740402222, 0.3884284496307373, -0.3906501531600952, -1.6781967878341675, -0.5947114825248718, -0.6610252857208252, 0.049261871725320816, -0.5974804162979126, -0.8744550943374634, -1.75554621219635, -1.1749321222305298, 0.4844377934932709, -2.9567902088165283, 0.19808077812194824, 0.8936654329299927, 0.3027312755584717, -0.32582810521125793, -1.0709270238876343, 0.32487303018569946, 0.9979193210601807, -1.1845574378967285, 0.16041861474514008, -0.8688637614250183, -0.741750955581665, 0.12613123655319214, 0.4446439743041992, 1.6988924741744995, 0.4680471420288086, -0.6368327736854553, 0.3807840347290039, 0.5219982266426086, 1.4159566164016724, -1.7103745937347412, 0.6937999129295349, 0.1115892305970192, -1.0318182706832886, 1.770321011543274, -1.7368438243865967, 0.2698715329170227, -0.016982948407530785, -0.06570900231599808, -0.4115453064441681, 2.5459628105163574, -1.0254658460617065, -1.053473949432373, -0.6346521973609924, 1.4188753366470337, -0.3132554292678833, 1.1469142436981201, -0.7857244610786438, 1.5409702062606812, -0.7490209341049194, -0.9099647402763367, 0.0648168995976448, 1.3579018115997314, -0.49426984786987305, -0.1942891925573349, -2.5694198608398438, -0.16788433492183685, -0.16740821301937103, 1.5622503757476807, -0.08667342364788055, 0.9729655385017395, 1.2331774234771729], [-0.016995498910546303, -1.2437548637390137, 0.6895349025726318, 0.5797113180160522, 0.4245743751525879, 0.8039274215698242, 2.233673572540283, -0.2597689628601074, -0.981579601764679, -0.1300196349620819, -0.6223623156547546, 0.6913101673126221, -0.9182893633842468, 1.2780396938323975, 0.06457460671663284, 0.9268682599067688, -1.5254908800125122, 0.874261200428009, 0.8001928925514221, -1.2993911504745483, -2.3202621936798096, -0.40592679381370544, 0.3555319607257843, 0.4545319974422455, 0.046127647161483765, -0.8055021166801453, -1.4353340864181519, -0.5424641370773315, 2.270643711090088, -0.33061376214027405, 0.3436678349971771, -0.4369509518146515, 1.0175753831863403, 0.5270349979400635, 0.03508279100060463, -0.342550128698349, 3.6885592937469482, -1.2851529121398926, -1.002076506614685, 0.5856118202209473, 0.5929150581359863, 1.724283218383789, -0.14406348764896393, 1.2947978973388672, -1.5091586112976074, -0.34068188071250916, -1.1696343421936035, -1.6042311191558838, 0.7551668882369995, 0.9006802439689636, 1.537635326385498, -0.2534732222557068, 1.2127872705459595, -0.2603648602962494, 0.1564028412103653, -0.3446884751319885, 0.606959879398346, 1.2651699781417847, 0.1826803982257843, 1.5569326877593994, 0.6806789636611938, -0.8387453556060791, -1.4029124975204468, -0.3475993871688843, 0.18055285513401031, 0.9821167588233948, -1.3972452878952026, 1.1857340335845947, 0.36366602778434753, -0.33572837710380554, 0.39755979180336, -0.20976436138153076, -0.37208428978919983, 0.8160885572433472, 1.121285319328308, 0.7162944674491882, 1.4296563863754272, 0.3494076728820801, -0.7718698978424072, -0.5354519486427307, -0.19283096492290497, 0.02296544425189495, 0.14406642317771912, 0.2037222981452942, 1.022544503211975, 0.9343753457069397, 0.8594290614128113, -1.176195740699768, -0.5516377091407776, -0.7194803953170776, 0.0739014595746994, 0.044263530522584915, 1.1149296760559082, -0.4973434507846832, -1.3288832902908325, -0.421133428812027, 0.964966893196106, 0.5462932586669922, -0.9434139728546143, 2.293980836868286, -0.7420359253883362, -0.15641488134860992, 0.15637142956256866, 0.8765034675598145, -0.6929896473884583, 0.24631260335445404, 0.49032625555992126, 0.8147328495979309, -1.6233525276184082, -0.5858139991760254, -0.7996561527252197, -1.1654582023620605, -0.7953095436096191, -0.32106393575668335, 0.7341778874397278, 0.46508780121803284, -0.7604038119316101, -0.4759874939918518, -0.46426743268966675, -0.42279183864593506, -0.2613750100135803, -0.2351352721452713, 0.4782887399196625, -1.0774219036102295, 0.8114495873451233, 0.6619845628738403, -0.993769109249115, 0.3905794620513916, 0.32974961400032043, -0.3951686918735504, 2.576627016067505, 0.4069349467754364, 0.520222544670105, -0.06323356181383133, 2.0898680686950684, -1.7435632944107056, -0.32874566316604614, -0.16291740536689758, 1.4913527965545654, -0.854813814163208, -0.35711556673049927, -2.246537208557129, -1.2366762161254883, 0.7837859988212585, 0.26447299122810364, 0.2041386514902115, 1.0183132886886597, 1.10699462890625, -1.362373948097229, 1.7015478610992432, 0.8068139553070068, -0.2538665235042572, 0.06983891129493713, -1.2836780548095703, -0.7760282158851624, 0.28829729557037354, -0.12369249761104584, -0.5229858756065369, -0.8580547571182251, 0.005214390344917774, -0.6198940277099609, -0.6735197305679321, 0.2836570143699646, -0.689337432384491, 1.0086628198623657, -0.6142944693565369, -1.429814100265503, 1.1264479160308838, 0.9823855757713318, -1.3039448261260986, -0.23480209708213806, 0.10509086400270462, -0.5572370886802673, -1.5007824897766113, 1.1250909566879272, -0.5329382419586182, -0.40630319714546204, -0.5483618974685669, 1.66396164894104, 0.9146422743797302, -1.2591657638549805, -1.754280686378479, -2.07340669631958, 0.6760189533233643, -0.9567447304725647, -0.7789495587348938, 0.11581974476575851, 0.5130764842033386, 0.5860039591789246, 0.9120061993598938, 2.0281968116760254, -0.5912599563598633, 0.8651159405708313, 1.9673748016357422, -1.155907154083252, -0.7962849736213684, -0.8752155303955078, 0.7025707364082336, 1.1717278957366943, 0.8623486161231995, -0.15138892829418182, -2.0967061519622803, -0.5676863193511963, 0.499035507440567, 1.3478801250457764, 0.8447412848472595, 0.4978524446487427, 1.9722263813018799, -1.1404674053192139, 0.5786027312278748, 0.6433339715003967, 1.2026193141937256, 1.0061513185501099, -0.1959577202796936, 1.4903042316436768, 0.5542244911193848, -1.4624335765838623, 1.4241048097610474, 0.5684698224067688, -0.10377167910337448, 1.1216124296188354, -0.4157641530036926, 0.3307439088821411, -1.3627821207046509, 0.09565918147563934, -0.8380365967750549, -0.9521440267562866, -1.1296515464782715, -0.355141282081604, 0.7679647207260132, -0.13223572075366974, 0.15460948646068573, 0.08655752241611481, 0.6402527689933777, 0.9189488291740417, 0.3399704098701477, -0.19148874282836914, -0.5046466588973999, 0.9464749097824097, -0.18111062049865723, 1.199629306793213, 1.5061558485031128, -0.7520954012870789, 0.7939744591712952, -0.10581855475902557, -0.7414847016334534, 1.2574975490570068, -0.23903867602348328, -1.0838340520858765, -0.13818298280239105, 0.5202322602272034, 0.25163573026657104, -0.46316519379615784, 0.20643679797649384, 0.4543571174144745, -0.23889347910881042, -1.8561419248580933, 0.9291579127311707, 0.5151758193969727, -1.8942160606384277, 0.46001937985420227, 0.29607266187667847, -0.29080265760421753, 0.5775947570800781, 1.3659920692443848, -1.1708062887191772, 1.1156963109970093, 0.16704539954662323, -0.49712127447128296, 0.2915079593658447, -0.15819548070430756, -1.166616678237915, 0.13446034491062164, 1.5551389455795288, 0.5129538178443909, -0.5743228197097778, 0.554792046546936, -0.27915066480636597, 0.04655645415186882, 0.1948394477367401, -0.9231740832328796, -1.3702493906021118, 0.034673094749450684, -1.0459511280059814, -3.60954213142395, 1.171600341796875, -0.9943611025810242, -0.14545927941799164, -1.2765668630599976, 1.2327730655670166, 0.27827587723731995, -0.020818425342440605, 1.1577845811843872, -1.179522156715393, 1.019827961921692, -2.8262975215911865, 0.47956153750419617, -0.22961808741092682, -1.729446291923523, 1.2866157293319702], [0.037283629179000854, 0.8539801239967346, -1.1216658353805542, 0.6416375041007996, -0.6453091502189636, 0.5913233757019043, 0.4775669276714325, 1.143804907798767, 0.7361211776733398, -0.30954068899154663, -1.7830390930175781, 1.2578572034835815, 1.3705452680587769, -0.06296265125274658, 0.23765477538108826, 0.3289586007595062, 1.6870429515838623, 2.10044002532959, -0.14435753226280212, -0.019520899280905724, 0.5079502463340759, -0.3229701817035675, -1.6936430931091309, -0.014177568256855011, 0.6753247976303101, -0.8882738351821899, -0.2528780698776245, 2.054903984069824, 0.6950174570083618, -0.4177650511264801, 0.13578170537948608, 0.8803545236587524, 1.3638290166854858, 1.4134577512741089, -1.9639911651611328, -0.5013901591300964, 0.5663341879844666, 0.9381933212280273, -1.8006789684295654, 0.4789217710494995, 0.18930721282958984, 0.9695823192596436, 0.9705417156219482, 1.327757477760315, -1.4705686569213867, 0.027760712429881096, -0.2897424101829529, 0.273508220911026, 0.1204281747341156, -0.042198918759822845, -0.30621132254600525, 0.12211575359106064, 1.3510197401046753, -0.026350565254688263, 0.3633587956428528, 1.6291435956954956, 1.8738614320755005, -0.16260455548763275, 2.024871826171875, -0.03954799845814705, -0.4719047546386719, -0.7289156913757324, -0.6419550180435181, -0.6404454708099365, -0.03567589074373245, -2.127204179763794, -1.117310881614685, 0.7639161348342896, -1.430319905281067, 0.817617654800415, -1.1320194005966187, -0.3074541985988617, -0.39609310030937195, 0.4368198812007904, 0.6471518874168396, 1.3487638235092163, -1.4654607772827148, -0.1267566978931427, -0.43288129568099976, -0.9449843168258667, 0.34339940547943115, 1.2960021495819092, -1.425467610359192, -0.44643843173980713, 0.37067490816116333, -0.2322128415107727, -1.219234585762024, -0.27654245495796204, -0.031782571226358414, -0.500127911567688, 0.9442148208618164, -0.0051897866651415825, -0.061027608811855316, 0.4676518142223358, -0.6733759045600891, -0.1934761106967926, -0.7909402847290039, -0.7004987001419067, -0.19745299220085144, 0.36059078574180603, 1.4698796272277832, -0.9183419346809387, 1.2656625509262085, -0.11412827670574188, -1.2146995067596436, 0.5032331943511963, -1.4945794343948364, 1.3345354795455933, -0.2703286409378052, -0.19610685110092163, 1.7868361473083496, 0.210239440202713, 1.0540025234222412, 1.1904082298278809, -1.1573883295059204, -0.7078102231025696, 0.1533169001340866, 0.4394083321094513, -0.5335103869438171, 1.2213369607925415, -1.3535915613174438, -0.08747808635234833, -1.584221601486206, 0.6980294585227966, -0.5759525895118713, -1.1004009246826172, 0.6585330963134766, 0.3943723440170288, 1.1992413997650146, -0.09862799197435379, 1.6328071355819702, -1.1033756732940674, -0.13219206035137177, -0.5964993238449097, -1.5526070594787598, 0.029831405729055405, 0.13168436288833618, 0.22757777571678162, -1.894579529762268, -0.09034735709428787, 0.6804555058479309, -1.7555299997329712, 0.6263206601142883, -1.8863049745559692, -0.8158257603645325, -0.3833317756652832, 1.0416334867477417, -0.22240562736988068, 0.7184473872184753, -0.4762358069419861, 0.27298763394355774, 1.054186224937439, 1.5180541276931763, 0.8766442537307739, -1.6463887691497803, 0.6908169388771057, 1.1391278505325317, 1.8206831216812134, -0.6349380016326904, 2.9520978927612305, 0.30279698967933655, 1.104205846786499, -0.6391832232475281, -1.1431950330734253, -1.5815767049789429, 0.1298278421163559, 0.3882148861885071, 1.4370307922363281, 0.6119147539138794, -0.1959962546825409, 1.5902456045150757, 0.4637777805328369, 0.0672958493232727, -0.4147852063179016, -1.2710058689117432, 1.5003806352615356, 0.2400072067975998, -1.4868192672729492, 0.26087433099746704, 1.1245346069335938, 1.3802399635314941, -0.8357725143432617, -1.6512537002563477, 0.23435519635677338, 1.157716989517212, -2.434346914291382, -0.2969529926776886, -0.039673954248428345, 0.10563170909881592, -0.8724688291549683, -0.5705265402793884, 0.6238225698471069, 1.3482248783111572, -0.383096843957901, 0.007605757098644972, -0.15703529119491577, -0.40152373909950256, -0.9311478734016418, 0.013928853906691074, -0.7632972598075867, 2.7380945682525635, 1.28177809715271, 2.65901517868042, 0.6085626482963562, 0.010833716951310635, 0.7366335988044739, 0.5237357020378113, 1.313528060913086, -0.27479758858680725, 1.4782359600067139, 0.9702653884887695, -0.4710046052932739, -1.2583941221237183, -0.469690203666687, -0.34597525000572205, -1.2188676595687866, -1.7249420881271362, 0.32134100794792175, 1.5181381702423096, 0.7943384647369385, 0.3152167797088623, -1.1409096717834473, -0.9854258894920349, -0.3621450364589691, 0.2253429889678955, -0.8237289190292358, -0.05288835987448692, 0.919050931930542, 0.6876362562179565, -0.8900482654571533, -0.673701822757721, -0.9083808064460754, -0.25673115253448486, 1.4452000856399536, 0.917346179485321, 0.6063010096549988, 0.279323935508728, 0.5927501320838928, 0.05915546417236328, 0.8719003200531006, 0.03899857774376869, -0.41080817580223083, 2.032127857208252, -0.1491362303495407, 0.23369044065475464, 0.36050865054130554, 0.9708890318870544, -0.7849729657173157, -0.20558997988700867, -0.5540890097618103, -0.25629979372024536, 0.24110518395900726, -0.2602395713329315, -0.6099706292152405, -0.25818297266960144, 0.9095894694328308, -0.11254557967185974, 1.3457000255584717, -0.583513617515564, -1.5635595321655273, 0.295741468667984, 1.0688027143478394, 1.166916847229004, -0.47194191813468933, 1.6056714057922363, -1.121363639831543, 0.6152785420417786, 2.472723960876465, -0.5137441754341125, -0.004363358486443758, -0.6842503547668457, -0.8452640771865845, -0.39975106716156006, 1.9290475845336914, 0.16827717423439026, 0.4147340953350067, -0.22521647810935974, 1.3598954677581787, -2.0870254039764404, -0.06298013031482697, 0.5393345952033997, 0.5009738802909851, 1.874929666519165, -0.35120689868927, -0.17786367237567902, 1.4457679986953735, 1.2058155536651611, 0.8024119138717651, 1.0690866708755493, -0.49182838201522827, -1.2319859266281128, 0.48291778564453125, 0.006087245419621468, -0.3055284321308136, -0.7462045550346375, -0.6724740862846375, -0.025847114622592926, 0.015067972242832184, 0.017702177166938782, 0.030769988894462585], [-0.611886203289032, 0.26174572110176086, 0.6252580285072327, -0.4331963360309601, 0.5263113379478455, -1.1391823291778564, -1.4108455181121826, -0.051884155720472336, -0.5912379026412964, 0.18412137031555176, -0.4117133915424347, -2.048105239868164, -1.9412922859191895, 0.5787409543991089, 0.2537083327770233, -0.9584422707557678, -0.25548040866851807, -0.3964059054851532, -0.37165817618370056, 0.5847029685974121, -0.23433703184127808, 1.1473196744918823, 0.10038499534130096, -1.2532317638397217, -0.6378400921821594, -1.6376460790634155, 2.4203760623931885, 0.008249865844845772, -1.5598894357681274, -1.8479423522949219, -0.7473160028457642, 0.008652844466269016, 0.5351994037628174, 0.30193910002708435, -1.3457729816436768, 1.651754379272461, -1.2730566263198853, 1.9769363403320312, -0.4321475028991699, -2.075960636138916, -0.32339999079704285, -0.24519026279449463, -0.20239339768886566, -1.2917267084121704, -0.1471235454082489, 0.5411810278892517, -0.21198025345802307, -1.299075722694397, 0.5600218772888184, -0.6029060482978821, -0.0826493352651596, -0.7965023517608643, -0.4982982277870178, 0.9338557124137878, -0.6936069130897522, -2.4112894535064697, 1.0811548233032227, 0.14241531491279602, 0.443647563457489, 0.13853713870048523, -0.4069593846797943, -0.13394910097122192, 0.8578796982765198, 1.0388906002044678, -0.7503917813301086, 0.04059874266386032, 0.4001957178115845, 0.7902817726135254, -0.49322885274887085, -0.8057076334953308, -1.4011956453323364, -0.3162078857421875, 0.16935564577579498, 1.3710094690322876, 0.816670298576355, -0.6484075784683228, -1.5560431480407715, -0.15350677073001862, 0.30701959133148193, 0.7139447927474976, -0.46218737959861755, 0.8188316822052002, -0.7930837273597717, 0.5984582901000977, 1.0689507722854614, 0.22746749222278595, -0.8623918294906616, -0.8209760189056396, 1.9689157009124756, -0.34877073764801025, -0.9023885130882263, -0.16365261375904083, -0.5969236493110657, -1.0781720876693726, 1.0259299278259277, -0.5223994255065918, -0.8425605297088623, -0.7888888716697693, 0.30689507722854614, 0.9106723070144653, 1.453527569770813, -0.5381367206573486, -0.651064395904541, 0.017165230587124825, 0.11000053584575653, -0.6247955560684204, -0.727451741695404, 0.6973499655723572, -0.9024438858032227, -0.654085099697113, -1.313431978225708, -0.5193987488746643, -0.0030885878950357437, 0.8346308469772339, 0.2105344831943512, -0.3844127953052521, 0.24909788370132446, 0.7348079085350037, 1.0216600894927979, -1.1514703035354614, 0.3383388817310333, -0.6428890228271484, 0.6267498135566711, 0.6348037719726562, -0.16958913207054138, 2.111711025238037, -1.0277483463287354, 0.724253237247467, 0.07565898448228836, 0.4128336012363434, 0.81348717212677, 0.39383092522621155, 0.08448687940835953, -0.4985711872577667, -2.2704577445983887, 1.9288926124572754, 0.06381972879171371, -0.6897677183151245, 0.11468859761953354, 0.296592652797699, -0.4138825535774231, 0.23896439373493195, 0.8894441723823547, -1.1434465646743774, 0.36630213260650635, 0.9131113290786743, 0.4783226549625397, -1.96310293674469, -0.18661542236804962, 0.831378698348999, -0.0014219173463061452, -0.5464298725128174, 2.2771482467651367, 0.819083034992218, 0.679302453994751, 0.8434024453163147, 1.0373305082321167, 0.47001200914382935, -0.09974118322134018, -0.5428611636161804, -0.912248969078064, -0.6666206121444702, 0.3665265440940857, -0.1407848596572876, 1.3756768703460693, -0.3474884033203125, 0.8086118698120117, -0.3649441599845886, -0.22957871854305267, -0.5934164524078369, -0.5510433316230774, 0.9543522000312805, 2.263446569442749, -1.2225409746170044, 1.8761842250823975, -0.9748536944389343, -0.36146488785743713, 1.648197054862976, -0.30033770203590393, -0.0011057091178372502, 0.09196850657463074, -0.43734440207481384, 0.6546094417572021, -0.6347014904022217, 0.2598245441913605, -0.7879993915557861, -0.9242457151412964, -1.6067347526550293, 0.8768707513809204, 2.4577293395996094, 0.7586527466773987, 1.1952803134918213, -0.2893602252006531, 0.07309642434120178, 0.4451523721218109, 0.1742859035730362, 0.19284141063690186, -0.03647207096219063, -0.10778407752513885, -0.3506791591644287, -0.7152496576309204, -0.6446437835693359, 2.1997199058532715, -2.95801043510437, 1.3567512035369873, -1.1970114707946777, -0.0701775923371315, -0.5868607759475708, -0.6294758319854736, -1.362525463104248, -0.482989102602005, -0.1609557569026947, -0.5242487192153931, -0.49763524532318115, -0.0075995647348463535, 1.269683599472046, 0.9115157723426819, -0.8210858106613159, -0.6563469171524048, 1.4626257419586182, 0.5276860594749451, -1.6824781894683838, 0.08634402602910995, 1.0524762868881226, 0.5724745392799377, -0.37820348143577576, -0.3452630341053009, 0.009315621107816696, 0.8745565414428711, -0.011576442047953606, 2.44808292388916, 0.6533548831939697, 0.2784343361854553, 1.3073378801345825, -1.441807508468628, 0.9006364345550537, -0.3446708023548126, -0.05676930397748947, -0.2751709818840027, -0.2837400734424591, 0.046305857598781586, -1.5926998853683472, 1.7530524730682373, 0.49810537695884705, 0.3548314869403839, 2.3326942920684814, -1.0780763626098633, -1.4879233837127686, 1.1078897714614868, -1.2865461111068726, 0.6264392733573914, -0.04579456150531769, 0.48347437381744385, -0.7566500306129456, 0.3294489085674286, 1.2468535900115967, 2.011575937271118, -1.2527270317077637, -1.5863319635391235, 1.9839791059494019, -0.464407742023468, 0.4148819148540497, 0.19549816846847534, -1.2403873205184937, -0.7402388453483582, 0.535716712474823, -0.2709043025970459, 0.7743460536003113, -1.3289722204208374, -0.49929630756378174, 0.1377985030412674, 0.0205577053129673, -0.4679112136363983, -0.6306259036064148, 0.23025807738304138, -2.0577805042266846, 0.5728939771652222, 0.22609111666679382, -0.6114673614501953, 2.1475002765655518, 2.612260103225708, -0.7254514098167419, -2.600820302963257, -0.0450269840657711, 1.143306016921997, 0.9518080353736877, 0.9078599810600281, 1.4143953323364258, 0.035263821482658386, 1.1283010244369507, -0.581320583820343, 0.4208689033985138, 0.47646182775497437, -0.052992962300777435, 0.26737040281295776, -0.24150605499744415, -1.1495167016983032, 0.3765508830547333, -0.10849779099225998, -0.5634070038795471], [1.0630264282226562, 0.6287528276443481, -0.37277981638908386, -0.7361994385719299, 0.10644178837537766, -1.285973310470581, 2.2729899883270264, -0.28653180599212646, 1.8833307027816772, -1.934192419052124, 0.713080883026123, -1.016769528388977, 0.5998318791389465, -1.1569137573242188, 0.593077540397644, -1.5243353843688965, -0.15259553492069244, -0.9925073981285095, 0.4597136378288269, -0.8780284523963928, -0.01841108687222004, -0.26483824849128723, -0.8370233178138733, -0.012870305217802525, 0.08302343636751175, -0.8997897505760193, -1.1841213703155518, -0.32649457454681396, 0.35622766613960266, -0.28836047649383545, -0.4828752875328064, 0.6111699938774109, -0.011069481261074543, 0.08507110923528671, 0.5563440322875977, -1.011266827583313, -0.9062872529029846, -0.2781773507595062, -0.7725329399108887, 0.9488669633865356, -0.7812081575393677, -0.3237588405609131, 1.2276294231414795, -0.4750179052352905, -1.4678400754928589, -1.0052542686462402, 0.7046005725860596, -1.7300152778625488, 0.30806756019592285, -0.4798026978969574, 0.7230659127235413, 0.18049530684947968, 0.7774304747581482, 0.7305075526237488, 0.5061120986938477, 0.09790810942649841, 1.2293413877487183, 1.0439964532852173, -0.5492090582847595, -0.5994400978088379, -0.6724522113800049, -0.6850895285606384, -0.5015583634376526, -0.5221776962280273, 1.0135374069213867, -1.1634541749954224, 1.1462239027023315, 1.3863325119018555, 1.1998169422149658, 0.03597627207636833, -1.7439968585968018, 0.8188824653625488, 1.3504765033721924, -1.0407963991165161, -0.750317394733429, -0.5984625220298767, -0.34770897030830383, 0.04751509800553322, -2.280985116958618, 0.7312536239624023, 0.011515035293996334, -2.1341171264648438, 0.9917178750038147, -0.5746923089027405, -0.9443179368972778, -0.12584179639816284, -0.07902105152606964, 1.0069085359573364, -0.19597163796424866, 0.12290563434362411, 0.5111287832260132, 0.8573110699653625, -1.1615519523620605, 0.9819519519805908, -1.8019158840179443, -0.10885167866945267, -0.9035841822624207, 0.09148149937391281, 0.7058579325675964, 0.5542420744895935, -0.4760141372680664, -3.0581696033477783, -1.3004719018936157, -1.3294328451156616, -0.8502252697944641, -0.5524860620498657, 0.06251388788223267, -0.25017741322517395, -0.42065462470054626, 0.2487049549818039, 2.4094862937927246, 0.4002149701118469, 0.05207455903291702, 1.3642115592956543, -1.1471420526504517, -0.08982923626899719, 0.07596253603696823, 0.7757387161254883, -0.8735141158103943, -0.292837917804718, 0.2872673273086548, 0.12627872824668884, -0.47666293382644653, 0.06521851569414139, 0.6245758533477783, 0.5148422718048096, -0.2613125741481781, 0.727319598197937, 0.06742413341999054, 0.8193013072013855, -0.7290395498275757, 1.1623021364212036, 0.41073572635650635, -0.08970455825328827, 0.8560620546340942, -0.22147265076637268, 0.5694487690925598, 2.4778287410736084, -0.5267636775970459, 0.14206081628799438, -0.8223136067390442, -0.5993185639381409, 0.26972585916519165, -0.33056822419166565, 0.44451385736465454, 1.9410741329193115, -1.1185486316680908, 1.861964225769043, 0.6057670712471008, 0.44895532727241516, 0.27287188172340393, 0.8494434356689453, -0.6104156970977783, -0.8027420043945312, -0.8689701557159424, -0.2498088777065277, 0.09982748329639435, 1.1565747261047363, -0.8732059001922607, -1.4944058656692505, 0.3890170156955719, -1.7406307458877563, 0.9343975186347961, -0.8946170806884766, -0.5753899812698364, 0.5870318412780762, 1.7013297080993652, -0.9874262809753418, -0.005910502281039953, 0.4479026794433594, 1.0940332412719727, 1.6723092794418335, 1.4130150079727173, 0.3950707018375397, -0.1650688648223877, -0.44564855098724365, 0.4810449779033661, 1.7860954999923706, 0.878415584564209, -1.072501540184021, -0.17156730592250824, -0.11768849194049835, -0.8727819323539734, -0.3762016296386719, -0.23254305124282837, -2.4558632373809814, 0.49759775400161743, -0.5325859189033508, -2.4949212074279785, -0.25237715244293213, -0.32195621728897095, 0.26347002387046814, 0.33609792590141296, -1.3596733808517456, 0.012615127488970757, -1.5079715251922607, 0.8416783809661865, 0.13095960021018982, -0.02835565246641636, -0.8533934354782104, -2.098658561706543, 0.4469321668148041, 1.040110468864441, 0.4427625238895416, -1.47347891330719, -0.8402809500694275, 1.8862335681915283, -0.17645873129367828, -1.2051684856414795, 1.4641553163528442, 0.9781252145767212, -0.9452255964279175, -0.49888381361961365, 0.9185881614685059, 0.035734549164772034, -0.2458183765411377, -1.2858047485351562, -0.22020778059959412, -1.5870857238769531, -0.900270938873291, 1.4870226383209229, -2.4883532524108887, 0.32960957288742065, -1.1373926401138306, -0.4068579375743866, -0.06686247140169144, 0.2323496788740158, 0.8141358494758606, -0.6116524934768677, -0.48795387148857117, 0.033683791756629944, 1.467545509338379, -0.6103602051734924, -0.4483325481414795, 0.6647277474403381, 1.2723701000213623, 0.929547905921936, 0.0500648207962513, 0.0662585198879242, 0.26190802454948425, 0.1711481213569641, 0.007646907586604357, 0.8647096753120422, 1.857067346572876, -0.08431629836559296, -0.2752072513103485, 0.2199890911579132, 0.34170153737068176, 0.7702946662902832, 0.35244572162628174, 0.8626617193222046, -1.3220607042312622, 0.10080641508102417, 0.9170743227005005, 1.070701241493225, 0.017823901027441025, 0.679069459438324, 1.0722577571868896, 1.8377842903137207, -0.6029769778251648, 0.7020845413208008, -0.6392844915390015, 0.12642532587051392, 0.4949001967906952, -0.6697319746017456, 0.3594690263271332, 0.40303727984428406, -0.9331346154212952, 0.015082523226737976, -0.3551384508609772, -0.7292304039001465, -1.1140106916427612, 1.0929054021835327, 1.507539987564087, 0.678782045841217, -2.1195192337036133, 1.4332212209701538, -1.2856261730194092, 0.04405669495463371, 0.11974450945854187, 0.7472543120384216, -1.912737250328064, 0.3433462977409363, 0.574302077293396, -0.49240753054618835, -0.48540112376213074, 1.146541953086853, -0.005552033893764019, -1.8928639888763428, -0.5573475956916809, 0.5529775023460388, 1.84590482711792, 0.29074931144714355, 1.759942889213562, -0.9657549262046814, -1.1251038312911987, -0.8314129710197449, -0.02031463012099266, 1.057706356048584, -0.18655088543891907], [1.41874098777771, -0.2785266935825348, 0.12953712046146393, 0.4161105751991272, 1.2530937194824219, -2.0403096675872803, 0.5691370368003845, 0.7734885811805725, -0.538066029548645, 2.1253998279571533, -0.7283797860145569, -0.3735259771347046, -0.5727436542510986, -0.3582629859447479, 2.0790486335754395, -0.23254066705703735, 0.6109063029289246, 1.8439984321594238, -0.17253874242305756, 0.8915579915046692, 0.7439618110656738, -0.0841020941734314, -0.5356921553611755, 0.12336167693138123, 1.6922165155410767, -0.05006870999932289, 0.9983606934547424, -1.5590846538543701, -0.3162858486175537, -1.0515124797821045, -0.254357248544693, 1.0474509000778198, -2.023542881011963, 0.6670187711715698, 0.5092405676841736, -0.2627084255218506, -0.2708210349082947, 0.6129195690155029, 0.7229729294776917, -1.1614916324615479, -1.321549892425537, -0.19644121825695038, 0.5226849913597107, 0.3240777850151062, -0.07447518408298492, 0.2932996153831482, 0.8142282366752625, 0.8570986986160278, 1.712969183921814, 0.1758728325366974, 0.05518893152475357, 0.280519038438797, -1.4352197647094727, 1.1035096645355225, -0.8143012523651123, -1.3864142894744873, 0.9306259751319885, 0.6469114422798157, -0.101414255797863, -1.30893075466156, 0.7258250713348389, 2.3775312900543213, -2.3410794734954834, -1.0582144260406494, 0.022998575121164322, 0.07980946451425552, -0.25813722610473633, 1.1784700155258179, -0.521232008934021, 0.9287662506103516, 1.151879072189331, 0.18129771947860718, 0.3488454222679138, -2.480323314666748, -0.002858655294403434, -1.1470136642456055, 0.2522355914115906, 1.246918797492981, 0.24508999288082123, -0.044840242713689804, 1.6779446601867676, 0.178988978266716, -0.08314022421836853, 0.9673131704330444, -0.30004051327705383, 2.083153247833252, -0.35083481669425964, 1.0388039350509644, -0.8062021732330322, -1.0135177373886108, 1.1763787269592285, -0.0702119991183281, -0.2694949805736542, 0.17897875607013702, -1.5930513143539429, 0.260849267244339, -1.0216524600982666, -0.06263580173254013, 0.7540315389633179, -0.34096112847328186, 1.7732939720153809, 0.6035441756248474, -0.5152363181114197, -1.150241732597351, -0.7983144521713257, 0.8946879506111145, 0.30956244468688965, 0.5534551739692688, 0.12115786969661713, -1.0874220132827759, 1.471204161643982, -0.36542773246765137, 0.45401662588119507, -0.19293347001075745, 0.21507462859153748, -0.3208792209625244, 0.09836846590042114, 0.5774425268173218, 0.4441082775592804, -0.6701685190200806, -0.013867405243217945, -1.5867103338241577, 1.446278691291809, 0.08628653734922409, 1.3035229444503784, 0.42688897252082825, -0.7016645669937134, -1.2856745719909668, -0.3410376310348511, -1.3018172979354858, -0.06031588464975357, -0.10075077414512634, -0.23202402889728546, 0.7280844449996948, 0.8779193758964539, -0.21642154455184937, -1.5082964897155762, 1.6036957502365112, 0.36347487568855286, 0.3310522139072418, -0.9116577506065369, -0.4250682294368744, -0.05721689388155937, 0.38961899280548096, 0.7984828948974609, -0.2768300771713257, -0.8820611238479614, 1.225114107131958, 0.6032009124755859, -0.973110020160675, -0.09716306626796722, 0.07867739349603653, 0.5034019351005554, 0.518442690372467, -0.605011522769928, -1.4377461671829224, -1.0349416732788086, -1.6644319295883179, -0.26617905497550964, 1.2821940183639526, 0.4297642409801483, 0.10034044086933136, 3.0497517585754395, -0.552793025970459, 0.2558296024799347, -1.7345608472824097, -0.4824090600013733, 0.10667138546705246, -1.2156471014022827, -1.8300637006759644, 0.34046852588653564, 0.4169803261756897, -0.026313278824090958, 1.1306533813476562, -1.559589147567749, -0.06396856904029846, -1.8342875242233276, -0.2939939498901367, -0.5584018230438232, -1.6229289770126343, 1.159659743309021, 1.081866979598999, -0.660738468170166, 0.24417974054813385, 0.4498419165611267, 0.6411816477775574, 0.513857364654541, -0.2618696987628937, -0.43045154213905334, 0.7074463367462158, 0.7353418469429016, 1.3447400331497192, 1.2272207736968994, -0.7207192182540894, 1.2389254570007324, -0.5903285145759583, -0.6405683159828186, 0.4089767336845398, -0.6983983516693115, 1.0315864086151123, 0.4209080934524536, -0.4823046624660492, 0.03989190608263016, 0.869723916053772, -0.2677558660507202, 0.48180559277534485, -1.1562858819961548, -0.28897738456726074, 0.1787247210741043, 0.2495432198047638, 1.5012567043304443, -0.0299854576587677, -0.10156891494989395, -1.7032616138458252, -0.8507981300354004, 0.35717785358428955, 2.0565953254699707, 0.02334146574139595, -0.0844731554389, -0.11590077728033066, -0.613864541053772, 0.11824946850538254, 0.3223772644996643, 1.8807003498077393, 0.007841511629521847, 0.49528658390045166, 0.9446832537651062, -0.689683735370636, -1.152735710144043, -0.164861261844635, 0.8153600096702576, 1.4193854331970215, 1.0639045238494873, 2.058382987976074, -0.6602592468261719, -0.10267091542482376, -1.2756614685058594, 0.9237461090087891, 1.6268665790557861, 0.5025368928909302, 0.08958281576633453, 0.2064840942621231, -0.7795915007591248, -0.9421665072441101, -0.29403188824653625, -0.10215182602405548, -0.35582953691482544, -1.1820327043533325, 0.4806227684020996, -0.27543336153030396, 0.12621718645095825, 0.9478163719177246, -1.3550251722335815, 0.9531071782112122, 0.7076699137687683, 0.15953339636325836, -0.3437059819698334, 1.6569777727127075, -0.6068335175514221, -1.2589951753616333, 0.5292755961418152, 0.878471851348877, 1.4524084329605103, 0.5349799990653992, 0.487306147813797, 1.5610299110412598, -1.68559730052948, 0.6497002243995667, -0.021194085478782654, -0.7248911261558533, -0.4076242744922638, 1.21171236038208, 1.9488848447799683, -0.24339602887630463, 0.1411300003528595, 0.09475406259298325, 0.9311630129814148, 2.975914716720581, -2.7201201915740967, 1.4714393615722656, 0.34670472145080566, 0.6440355777740479, -0.015293044038116932, -1.5600130558013916, 0.8685886859893799, 0.801982045173645, -0.42787426710128784, -1.0392649173736572, 0.17263442277908325, 0.9236605167388916, 0.09914349764585495, -0.5892660617828369, -1.2099477052688599, 0.43488889932632446, 0.4628995358943939, 1.1607085466384888, 0.3788659870624542, 0.44786712527275085, 0.995903491973877, -2.077209234237671], [0.5175004601478577, -0.820442795753479, -0.892344057559967, 0.5150932669639587, 0.5070293545722961, 0.18823467195034027, 1.274712085723877, -0.7248997092247009, 0.3517829179763794, -0.18722794950008392, 1.6610398292541504, -0.7843148708343506, -0.1270064115524292, 0.4541183412075043, 2.297398567199707, 0.882783055305481, 0.47499778866767883, 0.11445260792970657, 1.0663622617721558, 0.7174752950668335, -0.3408283591270447, 0.1274888515472412, 0.7890909314155579, -0.6703094840049744, -0.7070884108543396, 0.9849027991294861, 1.0598478317260742, -1.9849106073379517, 0.26897063851356506, -0.3352608382701874, -1.3094290494918823, 0.8927545547485352, -1.9330633878707886, 0.5156298279762268, 0.5712132453918457, -2.981210947036743, -0.08786341547966003, 0.46330079436302185, 0.10536283254623413, 1.9083685874938965, -0.8874019384384155, 0.24722248315811157, -1.2830188274383545, -0.3946402668952942, 1.3609592914581299, -0.2697383463382721, 2.0419583320617676, -0.41888290643692017, 0.4256782531738281, 0.3076289892196655, 0.6871801614761353, 1.2286555767059326, -0.6855810880661011, -0.391154944896698, -0.0888209417462349, 0.012287553399801254, 1.6210641860961914, -1.3469480276107788, -0.05042833834886551, -0.0921001136302948, 0.9186570644378662, -0.545499861240387, 0.28729507327079773, -0.09396539628505707, -0.38249868154525757, 0.4772954285144806, 1.3980485200881958, 0.5308123826980591, -0.27066460251808167, 0.06166819483041763, 0.23187342286109924, 0.756437361240387, -1.3133594989776611, -0.08755819499492645, -0.18774276971817017, 1.2508978843688965, -1.4425283670425415, -0.5646607279777527, -0.823691725730896, 1.7407342195510864, -0.3027176558971405, 0.5194019079208374, 1.0671565532684326, 0.35130974650382996, -0.4802212417125702, 1.382649302482605, -0.7878508567810059, 1.4015849828720093, 0.3305467367172241, 0.9850168228149414, -2.032331705093384, 0.7575303912162781, 1.8110522031784058, 0.037370648235082626, -1.5312601327896118, -0.2535942494869232, 0.33254262804985046, -0.41931936144828796, -0.8360225558280945, 0.15668155252933502, -0.2516809105873108, 0.6025698184967041, -0.38896238803863525, -0.11095927655696869, 0.3598175346851349, 0.7021803855895996, 0.8515095114707947, -0.3794877827167511, -0.10624834150075912, -0.28382232785224915, 1.5018905401229858, -0.3978124260902405, 0.0008218178991228342, -0.7129362225532532, -0.29565054178237915, -0.6368212699890137, -0.5456028580665588, 0.05294421687722206, 0.49786993861198425, 0.615343451499939, 0.1419738233089447, 0.5109696388244629, -0.1680750548839569, -0.06370219588279724, -0.2823043763637543, 0.4191252291202545, 0.028756573796272278, 1.8730281591415405, -1.2728792428970337, 0.0888049378991127, -1.016060709953308, -1.6594411134719849, -0.7252956032752991, -0.0030648731626570225, -0.9681990146636963, 1.1070548295974731, 0.20563440024852753, 0.30335038900375366, -0.9329586625099182, -0.026308774948120117, 0.3724370300769806, 0.47461622953414917, 1.5532560348510742, 0.042802534997463226, 0.4707663357257843, -0.14172472059726715, 0.6600384712219238, -0.03452164679765701, 0.08919627964496613, 2.117570400238037, 1.4812700748443604, -0.9678824543952942, -0.8843932747840881, 0.8010603785514832, -0.8822985291481018, -0.19898007810115814, 0.9543302059173584, -1.006685733795166, 0.8000075817108154, -0.10206036269664764, 0.18185879290103912, -0.4983442723751068, 0.31159707903862, 0.7369726896286011, -0.40045949816703796, 0.13142089545726776, -0.7704179286956787, 0.8037497997283936, -1.2051197290420532, -0.405471533536911, -1.3872486352920532, -1.3874688148498535, 0.04099585860967636, 0.8135356307029724, 0.1199905276298523, 0.3459223210811615, -1.0663753747940063, 0.40163692831993103, 0.439451664686203, -0.6343623399734497, 0.16450779139995575, 0.15120473504066467, 0.7257599234580994, -1.5998518466949463, -0.9482210278511047, -0.460227370262146, 0.15425050258636475, 0.30503571033477783, -0.5906684994697571, -0.804495632648468, 0.6642524003982544, 0.33287933468818665, -1.7288566827774048, 0.07806329429149628, 0.5003018379211426, 2.5007288455963135, 0.5799890160560608, 0.8949939012527466, -0.3644421696662903, 0.10524304956197739, -0.08742725104093552, 0.29102760553359985, 0.29257732629776, -0.4455355107784271, 0.2860841751098633, -0.7969266772270203, 0.11242800205945969, -0.20719805359840393, -1.1396870613098145, 0.3077443242073059, 0.6090501546859741, -0.5969297885894775, 0.2150149643421173, -0.10659516602754593, -1.9061713218688965, -0.24752379953861237, -0.8559314608573914, -0.07353945076465607, -0.42983707785606384, -1.7515581846237183, -0.23713074624538422, -1.222184658050537, -0.30284780263900757, -0.1615247130393982, -0.34993410110473633, 0.7873873710632324, -0.2914618253707886, 0.32233983278274536, 0.9236733913421631, 0.897560715675354, 1.6092857122421265, 0.5801299810409546, -1.6762096881866455, -0.30400848388671875, -1.6638113260269165, 0.6392528414726257, 0.29744425415992737, -1.3437888622283936, -0.6567590832710266, 0.7732627987861633, 0.34609970450401306, -0.41154757142066956, -1.6492570638656616, 0.608856201171875, 0.2000521421432495, -0.7696537971496582, -1.470800757408142, -1.0205047130584717, 2.7883663177490234, 0.4905841648578644, -1.2169017791748047, -0.6115469932556152, -0.7070336937904358, 1.3270394802093506, 0.23450419306755066, -0.7076351046562195, -0.08485299348831177, -1.1492301225662231, 1.8421151638031006, 0.4081750214099884, 0.09626755863428116, 0.006815794855356216, -0.3304356038570404, -1.2632941007614136, -0.5260230898857117, 1.4997162818908691, -0.3470057249069214, 1.246790885925293, -0.4256958067417145, -0.002749649342149496, -1.4496502876281738, -1.2286673784255981, 0.053113676607608795, 0.9402866959571838, 0.35608038306236267, -1.5591981410980225, -1.053971767425537, -0.7882444262504578, 1.297969937324524, 1.0676435232162476, -1.5066503286361694, 0.2911909222602844, -1.3561902046203613, 1.621295690536499, 1.177675724029541, -1.252152681350708, 1.3876392841339111, -1.135380744934082, 0.7699540257453918, 0.012603155337274075, -0.6093989014625549, -0.5771627426147461, -0.07538893818855286, 1.571155071258545, 0.02467983029782772, -0.06524341553449631, -0.8443301320075989, 1.2665728330612183, 0.44846755266189575, 0.5387206673622131], [-0.44904735684394836, -0.08707261085510254, -0.15886521339416504, -0.7915688753128052, 0.09815634787082672, -0.35601359605789185, -0.3241252601146698, 1.609094500541687, -0.6989830732345581, 0.20051774382591248, 1.2705191373825073, -0.39020252227783203, -0.7539868950843811, 1.255838394165039, 1.17292058467865, -0.2772250473499298, -0.443588525056839, -0.7923410534858704, 0.8634594678878784, 0.03351820632815361, -0.587460458278656, -0.1772865206003189, -1.5318365097045898, -0.23555004596710205, 0.4136961102485657, -0.683386504650116, -0.34571918845176697, -0.7614436149597168, -1.0769602060317993, 1.6513036489486694, 0.2446717768907547, -1.0452373027801514, -1.129028081893921, 2.2612900733947754, 0.49838000535964966, 1.1319317817687988, -0.20007330179214478, 0.5538725256919861, -2.7957026958465576, 0.8778309226036072, 0.2555962800979614, -0.8503614664077759, 1.1268254518508911, 0.5605542063713074, -0.8333140015602112, -0.31372472643852234, -0.3006238341331482, 1.690947413444519, -0.3806891441345215, -1.6382049322128296, 0.2187618911266327, 0.13384492695331573, 1.483152151107788, 0.8851949572563171, -0.06640200316905975, 0.4127175509929657, -1.0770294666290283, -0.06893050670623779, -0.2612878680229187, 0.9459527134895325, -1.405972957611084, 0.23060564696788788, 1.1237707138061523, -0.7831205725669861, 0.16937808692455292, 1.0929676294326782, 1.5756797790527344, 0.46680009365081787, 1.9144480228424072, -0.4767116904258728, -0.10034992545843124, -0.2254786491394043, 1.148425817489624, -0.989389181137085, -0.1990857869386673, -1.3512821197509766, -0.24517354369163513, 1.210418701171875, -0.27634182572364807, 2.035113573074341, 1.592785358428955, -0.17480942606925964, -2.0649874210357666, -0.5458113551139832, 0.22814927995204926, 1.2933400869369507, 0.783577024936676, -0.7558844685554504, 0.4073229432106018, -1.531811237335205, -0.9929861426353455, -1.224897861480713, 0.9512742161750793, -0.12583158910274506, 0.30182793736457825, 1.0297725200653076, -0.5202590227127075, -0.8277698755264282, -0.6533532738685608, 0.6989343762397766, 0.03508344292640686, -0.1777271181344986, 0.3685332238674164, 0.3210445046424866, -0.524914026260376, 0.06290146708488464, 0.5306228399276733, 0.15165017545223236, -1.0483741760253906, -1.08579683303833, 0.24730490148067474, 0.12560208141803741, -0.08811222016811371, -1.136675477027893, 1.7271356582641602, 0.6092530488967896, 0.2754000723361969, 0.5843262672424316, -0.8103967308998108, -0.7906966209411621, -0.9808176159858704, -0.500361442565918, 0.16570110619068146, -0.5799707174301147, -0.3083173632621765, -0.5014203786849976, 0.702815592288971, 0.00048116050311364233, -0.3307885527610779, -1.103620171546936, -2.0310986042022705, -0.7455465793609619, 0.2101670205593109, 0.587706446647644, -0.8881652355194092, -0.1661260724067688, 0.9700179100036621, 0.2907033860683441, 0.49644842743873596, -1.4973112344741821, 0.12153065949678421, -0.45819762349128723, 0.6522196531295776, 0.514678418636322, -1.0550040006637573, -0.2784249186515808, -0.23360836505889893, -0.28679290413856506, -0.12309030443429947, -0.7570669054985046, 0.8820700645446777, -1.583080530166626, -1.8785637617111206, 1.6683416366577148, -0.7257394790649414, -0.3406650424003601, 2.1294827461242676, -0.6103343367576599, -0.2468433827161789, -0.6796956658363342, 0.4707961082458496, -0.157478928565979, 1.4156591892242432, 0.6180022358894348, 0.22710120677947998, 0.1957961916923523, 0.683593213558197, -0.7992767691612244, -1.5517510175704956, -0.2776283025741577, 1.2268980741500854, 0.047767624258995056, -0.878347635269165, -0.08927413821220398, -0.2286260575056076, 1.4548108577728271, -0.16328132152557373, -0.4579753577709198, -2.799940586090088, -0.8020973801612854, 1.867591142654419, 0.7255990505218506, -0.7464118599891663, -0.7575171589851379, 0.4968617558479309, 1.0675220489501953, -0.048671234399080276, 0.6232072114944458, -0.21385008096694946, 0.3689171075820923, -0.5271437764167786, 0.6237014532089233, -1.2986178398132324, -2.487443208694458, 0.4700286388397217, -1.5956988334655762, 0.6417527794837952, -1.4158923625946045, 1.060442328453064, -0.34657686948776245, -0.3346940577030182, 0.286763995885849, 0.7837950587272644, -0.7438691258430481, -1.0190610885620117, -0.8091405034065247, 1.685836911201477, -0.31033381819725037, -1.2737784385681152, -1.4701074361801147, -0.08810988068580627, 1.37553071975708, 1.2792837619781494, -0.701949954032898, -0.6595274806022644, 0.22206658124923706, -1.5042697191238403, 0.3959346115589142, -1.1557592153549194, -0.07754436880350113, -0.35521888732910156, -0.862461268901825, -2.4580368995666504, 0.49971869587898254, 0.8327070474624634, 0.6391683220863342, -0.6058415770530701, 0.46940940618515015, -0.8672088384628296, 0.18632560968399048, 0.4215970039367676, 0.7402725219726562, -2.0087788105010986, -0.5466796159744263, 1.1248701810836792, 0.1294264793395996, -0.6778828501701355, 0.615311324596405, -1.7913836240768433, 0.045480936765670776, -1.5760929584503174, -0.3571244776248932, -1.9841208457946777, -0.34906667470932007, -0.03540941700339317, -0.07937625795602798, -1.4695227146148682, -0.13400352001190186, 1.0451761484146118, 1.0723373889923096, -0.9572731852531433, 0.4805983901023865, -0.5655276775360107, -1.0729007720947266, 0.44191914796829224, -1.2056564092636108, 1.8124555349349976, 1.3958276510238647, -0.6186907291412354, -1.4868236780166626, 0.21383199095726013, -0.9187567234039307, -0.8516804575920105, -1.6765207052230835, 0.4427623748779297, -1.0494376420974731, -0.00434874976053834, 1.2607810497283936, 1.136810541152954, -0.7472827434539795, 0.7559065818786621, -0.3727824091911316, -0.0010080047650262713, -0.5510154366493225, 1.8015401363372803, -1.4046733379364014, -0.1875360906124115, 0.33033642172813416, 0.12536099553108215, -0.14954759180545807, 1.990403652191162, -0.9315811991691589, -1.0834306478500366, -0.08755847066640854, 0.36514103412628174, -0.5590847730636597, -0.5119240283966064, 0.09418106824159622, -0.06025325506925583, -0.6914091110229492, 0.14407207071781158, 0.0008480363758280873, -0.12954643368721008, 0.6011167168617249, 0.534294843673706, -0.5469542145729065, 0.17929735779762268, -1.0548710823059082, -0.9264274835586548, -1.1346240043640137], [0.1556953340768814, -0.6551399827003479, 0.6824647188186646, -0.7514282464981079, -0.5798139572143555, 0.8318406939506531, 0.03806709870696068, -0.5527268648147583, 0.8109642267227173, 0.7394840121269226, -0.6003860831260681, 1.8521028757095337, 1.6129398345947266, -0.8738336563110352, 1.500747561454773, -0.7883356809616089, -0.2209983617067337, 0.20901058614253998, -1.6653789281845093, 1.090975284576416, -0.995729923248291, -1.0283337831497192, 0.42336997389793396, -0.40160632133483887, -0.372953325510025, -0.7760718464851379, -0.28244709968566895, -1.4429908990859985, 0.2401847094297409, 1.4290443658828735, -0.6904473900794983, 0.45297104120254517, -1.394929051399231, -1.6425353288650513, 0.45575785636901855, 0.11972933262586594, -0.4410318434238434, 0.9820964336395264, 0.7833365201950073, 0.36938294768333435, -1.909850001335144, -0.04047457501292229, -1.2329405546188354, 0.18075598776340485, 0.29123255610466003, 0.9812805652618408, 0.031897809356451035, -0.1658821552991867, 2.389303207397461, -0.1439005583524704, -0.607249915599823, -0.5248969197273254, 1.5273867845535278, 0.17152488231658936, -0.3612068295478821, 2.589538335800171, 1.1226012706756592, -0.4920642673969269, -0.09378498792648315, 1.1633859872817993, 0.6507538557052612, -0.5591586232185364, 1.2866133451461792, 1.2278889417648315, 0.17544785141944885, -0.9729406833648682, -0.48020556569099426, 1.0771560668945312, 0.9785252809524536, 1.1496427059173584, -0.6855980157852173, -0.18474291265010834, -2.452880859375, 0.9356100559234619, 0.4494817852973938, -1.2551393508911133, -0.5346456170082092, -0.7661890387535095, 0.9121786952018738, -1.5601905584335327, -0.43134650588035583, -0.0700671449303627, -0.6243929266929626, 0.73549884557724, 1.5338244438171387, 0.14474564790725708, -1.3510158061981201, -1.4935251474380493, -0.684390127658844, -0.5446408987045288, -0.5318350195884705, -0.1430720090866089, -1.5888603925704956, -0.05387812480330467, -0.2878965139389038, 0.7494333386421204, -1.627446174621582, -0.5192165970802307, 1.5741400718688965, 1.3012758493423462, -0.19280999898910522, 0.49487876892089844, -0.42698460817337036, 0.6103693842887878, -0.02851911634206772, -1.2542716264724731, 0.1101389229297638, -0.3377605080604553, 0.4170028865337372, -1.0775845050811768, 0.1030825674533844, 0.46502289175987244, 0.34926435351371765, -1.5749797821044922, -0.4295162856578827, -1.0942912101745605, 1.9011090993881226, 0.4767126739025116, -0.17635220289230347, -0.7777733206748962, 1.1904706954956055, -1.0084547996520996, 0.8864997625350952, -1.4477148056030273, -0.9257418513298035, 1.635582685470581, 0.10676243156194687, -1.635963797569275, 1.151445984840393, -1.0473850965499878, 1.1891145706176758, -3.237766981124878, -2.071207046508789, -1.5412081480026245, 0.08484547585248947, 0.08545496314764023, -0.5428235530853271, -0.2645402252674103, -0.6365337371826172, -0.27249792218208313, 0.43553805351257324, -0.0550505705177784, -0.8810364603996277, -2.069551706314087, -1.8900395631790161, -0.1688947081565857, -0.19858399033546448, -0.698002815246582, 1.4876158237457275, -0.1626022905111313, 0.1940016746520996, -0.6920899152755737, -0.3411059081554413, 0.019742881879210472, 0.05244957655668259, 0.04300420358777046, 0.1037241667509079, 0.6993187069892883, -0.06898976117372513, 0.8848560452461243, -0.41823554039001465, -0.5937532782554626, 1.9942504167556763, 1.0574312210083008, 0.48151272535324097, 0.061593782156705856, -1.3278307914733887, 0.6681288480758667, -2.3840227127075195, 0.07686369866132736, -1.0462522506713867, -1.1363801956176758, -0.6191651821136475, -0.8187050819396973, 0.33844804763793945, -1.1711466312408447, 0.2725085914134979, -0.907927930355072, -0.7234688997268677, -0.764516294002533, 0.5317587852478027, 3.099343776702881, 0.7485520243644714, -0.49771031737327576, -0.756401777267456, -0.8523520827293396, 1.0901867151260376, -0.203468918800354, 0.7300035357475281, -1.251960277557373, 1.5484641790390015, -0.5470146536827087, -1.0800224542617798, 0.18644042313098907, -1.1479904651641846, -0.9376054406166077, 1.3357023000717163, 0.8704686164855957, -1.6677848100662231, 1.237527847290039, 0.5698662996292114, 0.4607864320278168, 1.7684193849563599, -2.198763847351074, -0.14000695943832397, -1.3531126976013184, 1.9910786151885986, 0.4909288287162781, -1.7276663780212402, 0.7387683987617493, -1.0321940183639526, -0.054907191544771194, -1.1266449689865112, 1.714290738105774, -0.24790629744529724, -0.9649279117584229, 0.5157812833786011, -0.02887381799519062, -0.09437290579080582, -0.14454717934131622, -1.213258147239685, -0.5993077754974365, 1.3567485809326172, -2.3339555263519287, -0.33685094118118286, 0.1514289379119873, 1.829546332359314, 0.7687389254570007, 0.03919170796871185, 1.618385672569275, 1.0073953866958618, -1.157837152481079, 0.7712240219116211, -0.23435717821121216, 0.4531768262386322, 0.7065847516059875, -0.5180790424346924, 1.4459372758865356, 1.116729974746704, -1.1057217121124268, -1.3696889877319336, -1.4291434288024902, -1.137329339981079, 0.811648964881897, -2.225485324859619, 0.13118723034858704, 0.2360525280237198, 0.6472755670547485, -1.3504809141159058, 1.9989469051361084, -0.4510904550552368, -0.8802947402000427, 0.8564455509185791, -0.6457801461219788, 1.521927833557129, -0.21352320909500122, 0.3622094392776489, -0.5970315933227539, 0.9111558794975281, 0.0499657578766346, 0.44930607080459595, -1.4140472412109375, 2.4639062881469727, -0.31609222292900085, -1.3820629119873047, 0.8753998875617981, 1.0198241472244263, -1.8415272235870361, -0.610856294631958, -1.131213903427124, 0.5934357047080994, -1.3698137998580933, -0.8184651136398315, -0.6172451972961426, -0.27483344078063965, 0.16459426283836365, -0.3144174814224243, -0.46502017974853516, 0.6770134568214417, -0.7775970697402954, 0.5218513011932373, -0.22152619063854218, 0.07812197506427765, -1.1473371982574463, -0.5725460052490234, -0.23020340502262115, 0.11513693630695343, -2.6461071968078613, 1.703285813331604, 1.2377578020095825, 0.18172600865364075, -0.06800657510757446, 0.24647730588912964, -0.6933737397193909, -0.13909879326820374, 1.7049787044525146, -1.5916056632995605, 0.3226063549518585, -0.03807249665260315, -0.9030250310897827], [-0.04078497737646103, 1.4422892332077026, -1.7482469081878662, -2.66635799407959, -0.11294002085924149, 0.0008182913297787309, -0.7966923117637634, -0.7331435680389404, 0.6100746393203735, -2.931211471557617, -1.0140641927719116, 0.48499172925949097, -0.9133708477020264, -0.6360654234886169, -0.6984519958496094, -0.8405902981758118, -0.46009954810142517, -0.2860119938850403, 0.4194764494895935, 1.1444175243377686, 0.4050256013870239, -0.594730019569397, -0.08367819339036942, 1.2799038887023926, -0.463867723941803, -0.25679802894592285, -1.0767111778259277, 0.9268479943275452, 0.570140540599823, -0.6492285132408142, 0.5437461733818054, 1.1841920614242554, -0.09514587372541428, 0.4889712929725647, -0.4844242036342621, -0.2795095145702362, -0.44241416454315186, 1.222723364830017, 0.8065910935401917, 0.6468701958656311, -0.11945952475070953, 1.7193071842193604, 0.3432246446609497, -0.9070403575897217, -0.4206157922744751, 0.3321128487586975, -0.9058020710945129, 1.8564393520355225, -0.3119542598724365, -0.7081362009048462, -0.07916948199272156, -0.4285576045513153, -0.38060393929481506, 0.7815931439399719, -0.6528663635253906, 1.2893519401550293, -0.008732197806239128, 1.40282142162323, -1.0337145328521729, 0.0658668726682663, -0.8750643134117126, -1.3364245891571045, -1.9377838373184204, -0.9969758987426758, 0.46984192728996277, 0.9761264324188232, 0.2728947401046753, -0.7262672185897827, -0.18839675188064575, 0.20265334844589233, -0.5406628847122192, 0.7799782752990723, 0.9117342829704285, -0.35299673676490784, 0.02760939858853817, -0.7319861650466919, 1.1449437141418457, -0.1686282604932785, 2.5231871604919434, -1.3986060619354248, 2.1201932430267334, 0.07642437517642975, -1.5026029348373413, -1.0781546831130981, 1.079671859741211, -0.15889351069927216, 1.1569567918777466, -1.2864644527435303, 0.20060332119464874, -0.5578980445861816, 0.02490214630961418, -0.21731926500797272, 0.6083101630210876, 0.280994713306427, -0.5768176913261414, 1.6383627653121948, -1.581088662147522, 0.36116552352905273, -0.37117308378219604, -1.4823848009109497, -0.07594157010316849, -1.6063194274902344, -1.0260732173919678, -0.5712911486625671, 0.3954594135284424, 0.966020405292511, 1.0749907493591309, 1.126876950263977, -1.5316708087921143, -0.6750971078872681, 1.3261523246765137, 1.2173542976379395, -0.9304794669151306, 0.18862907588481903, -1.5240989923477173, -0.30048662424087524, 1.309052586555481, -0.023666737601161003, -1.1820168495178223, 1.8154171705245972, 0.1699933111667633, 0.1849287897348404, 0.6172850131988525, 0.6992366313934326, 1.3683671951293945, -0.37891244888305664, -0.5665183067321777, -0.5467405319213867, 2.080193042755127, 1.486451506614685, 1.1586718559265137, -1.376922607421875, -0.8928756713867188, -0.10369858145713806, -0.5595176815986633, -1.7370871305465698, 0.6294168829917908, -0.7289540767669678, -0.48521688580513, 2.2807364463806152, -0.47212743759155273, -1.1021404266357422, -1.8735558986663818, -0.18835695087909698, -0.8065804243087769, 0.602981448173523, -0.6214138865470886, -0.1452922374010086, 1.1350599527359009, 1.2734243869781494, 0.7171232104301453, 0.26258280873298645, -0.655415952205658, 0.37881648540496826, 0.04274279624223709, 0.3136965036392212, -1.1629682779312134, -1.4869458675384521, 1.3733463287353516, 0.8647752404212952, 0.3873487114906311, -0.06036819890141487, 1.5034997463226318, -1.632031798362732, -0.6834865808486938, -1.4828712940216064, 0.2474151998758316, -1.1411372423171997, 1.548394799232483, 0.8919617533683777, 0.33545035123825073, 1.088613748550415, -0.11940180510282516, 1.8623771667480469, -1.3034305572509766, -0.8930081129074097, -0.39790481328964233, 1.3971041440963745, 0.5803496837615967, 0.47608116269111633, 0.9221664667129517, -2.1298341751098633, 0.7881287932395935, 0.20342060923576355, 0.020802084356546402, -1.0954186916351318, -0.5607576370239258, -3.7475738525390625, -0.24122744798660278, -1.1436562538146973, -0.23307351768016815, -1.0196491479873657, -0.5436339378356934, -0.37463894486427307, -0.7918692827224731, 0.7073915600776672, -2.1154580116271973, -0.06827975809574127, 0.8818268179893494, 0.07122122496366501, 1.6519640684127808, 1.3441407680511475, -0.2887580990791321, 0.6383963823318481, 0.32175591588020325, 0.005438871216028929, 1.5977927446365356, -0.04259953275322914, 0.8033517003059387, 0.4002591073513031, 0.8247902989387512, 0.7554492950439453, -0.7552011013031006, 0.4454900026321411, 0.6736647486686707, 1.4975708723068237, -0.2585289180278778, 1.6708999872207642, -0.09114164113998413, -0.7073876261711121, 2.63126802444458, -0.8519821166992188, -0.2188945710659027, -0.8341268301010132, 1.9580110311508179, 2.1069986820220947, 1.4593982696533203, 0.5970974564552307, -0.07785675674676895, -0.1410263180732727, 1.0068594217300415, -0.5650942325592041, -0.20657339692115784, -0.12052662670612335, -1.2584360837936401, 0.8111050128936768, -0.7004271149635315, 0.2662339210510254, -0.36650875210762024, -0.594529390335083, -0.5702758431434631, -1.458496332168579, -0.32688480615615845, -1.0915570259094238, -0.2743888795375824, -1.1910699605941772, 0.10956517606973648, 1.033871054649353, 2.0778071880340576, 1.0144214630126953, -0.6317015886306763, -0.06288114935159683, -1.3023253679275513, -1.4575124979019165, -0.27773281931877136, -0.8593080639839172, -0.26793602108955383, 0.8561130166053772, 1.1210944652557373, -0.47673147916793823, -1.3187988996505737, -0.7216561436653137, 1.1134421825408936, -0.6451442241668701, -0.5898046493530273, -0.7287421226501465, 1.2627811431884766, 1.889825701713562, 0.13923998177051544, -1.7773351669311523, -0.3228546977043152, 1.8008532524108887, -1.6996387243270874, -1.2204777002334595, 0.3764732778072357, -0.1116405725479126, 0.9875238537788391, -0.35764580965042114, 0.9059653282165527, -0.2306097149848938, 0.15969929099082947, 1.4740409851074219, -0.801893413066864, 0.10613467544317245, 0.3907362222671509, 0.7470396757125854, -0.982250988483429, -0.6481962203979492, -0.6938953995704651, 0.47806692123413086, 0.7541041374206543, -0.946084201335907, 0.6722887754440308, -1.2227890491485596, 2.5732061862945557, -0.7952756881713867, -0.9356251358985901, -0.4241958260536194, 0.0351683683693409, -0.7632943987846375], [0.6399634480476379, 0.4799017906188965, 0.08872641623020172, 1.9096646308898926, -0.4410439431667328, -0.7866275906562805, 0.5041466951370239, -1.2329225540161133, 1.7442872524261475, -2.014272689819336, -1.5349094867706299, -0.01745879277586937, -0.16971273720264435, -0.33046308159828186, -0.8776456713676453, 1.4864616394042969, -0.4120985269546509, 0.19170837104320526, -1.3491275310516357, -0.8037112951278687, 0.5322847962379456, 0.6706830263137817, 0.6112877130508423, 0.4073280692100525, 0.6864993572235107, -1.9871315956115723, 0.2860071361064911, 1.0314421653747559, -0.08965759724378586, 1.3140294551849365, -1.0485793352127075, -0.7085145115852356, -0.40626248717308044, 0.14894098043441772, -0.7355626225471497, -2.1802597045898438, -0.3224082589149475, -0.0462355874478817, 0.9153313040733337, -0.8394407629966736, 1.3523738384246826, 0.37624940276145935, -2.2641048431396484, -1.1016616821289062, -1.1143378019332886, -0.07840815931558609, 1.2442082166671753, 0.8466609120368958, 0.9427381753921509, -0.5556969046592712, -0.09372573345899582, 0.18632525205612183, -1.5871860980987549, -1.5335559844970703, -0.6446908116340637, 1.2632828950881958, -0.2805750370025635, 0.9706831574440002, 1.876485824584961, 1.1738591194152832, -0.09962207823991776, -0.28537535667419434, 1.2472188472747803, -0.867467999458313, -0.21861779689788818, -2.1939690113067627, -0.6881422400474548, 0.014283744618296623, -1.043548822402954, -0.12442304939031601, 0.15648536384105682, 0.1548343449831009, 0.1623663753271103, -0.6403265595436096, 1.5817700624465942, 0.4428624212741852, -0.2731502056121826, 1.0500028133392334, -1.106472373008728, 1.313458800315857, -1.3683215379714966, 0.5749115943908691, -1.116974115371704, -0.7869331240653992, -0.6277087926864624, 0.3320411741733551, -0.05324062332510948, -0.4453399181365967, -1.4901081323623657, 0.9867497682571411, -0.12720364332199097, 0.2833186089992523, 0.8161795139312744, 0.31070733070373535, 0.7619472742080688, 1.62544846534729, -0.3376803696155548, 0.6192848086357117, 1.070358395576477, -0.34785109758377075, -1.3726637363433838, 0.862810492515564, -0.9069694876670837, 0.6728955507278442, -0.0856412872672081, -1.1893701553344727, -0.6809269785881042, 0.02184603549540043, 0.2788902223110199, -0.4532305896282196, 0.19823198020458221, -2.182933807373047, 0.4909716844558716, -0.682201623916626, 0.049647144973278046, 1.4183727502822876, 0.3488345444202423, 0.4623672664165497, -2.2066147327423096, 0.17596715688705444, 0.2258327305316925, -0.3068350553512573, -1.198284387588501, 1.364335536956787, -1.2694683074951172, -0.8618031740188599, 1.8882478475570679, -1.3560327291488647, 0.491952508687973, 0.824189305305481, -1.68712317943573, 2.202641010284424, 0.7456251382827759, -0.3215855062007904, 0.29380711913108826, 0.5954468250274658, -1.2294316291809082, 1.1800850629806519, 1.8152450323104858, -0.865839958190918, -0.2215850055217743, -0.2586364150047302, -1.3718501329421997, 1.4395936727523804, -0.4679023325443268, -0.23328661918640137, 0.05534522980451584, -0.33317190408706665, -1.632243275642395, 1.189691424369812, -3.1907079219818115, 0.5285266041755676, 1.5992026329040527, 0.9027416706085205, 3.1207797527313232, 0.43775850534439087, -0.5714625120162964, -0.19615887105464935, -1.319902777671814, -0.34055295586586, 0.4272807240486145, -0.25057220458984375, -0.03842705860733986, 1.5690802335739136, -0.11182474344968796, 0.4725117087364197, 1.6331782341003418, -0.6204873919487, 0.6933318376541138, 0.18141275644302368, 1.2511019706726074, -0.5672744512557983, -1.6576699018478394, -0.17343521118164062, -0.7049548625946045, 0.5608161091804504, -0.03278550133109093, 0.6678320169448853, -0.5357924103736877, -0.2977750301361084, 2.2825520038604736, -1.5744303464889526, -1.7812529802322388, -0.7001579999923706, 0.3233909606933594, 1.5317503213882446, -0.9729613661766052, -0.3793526291847229, -0.0016081681242212653, -0.20629088580608368, -1.4499244689941406, -0.8456724286079407, 1.473434567451477, 0.24670235812664032, 0.4012530744075775, 0.39697420597076416, 1.041573166847229, 1.619317650794983, -0.14856533706188202, 1.9346305131912231, 0.5717180371284485, -0.5700967311859131, 1.2528032064437866, 0.7185351848602295, -1.2588543891906738, -1.04074227809906, -0.3415753245353699, 0.3367917239665985, 0.47685009241104126, -1.7916808128356934, 0.8862360119819641, 0.27068808674812317, -0.9645452499389648, -1.3980286121368408, 0.13680389523506165, -0.42725256085395813, 0.662746250629425, 1.3336726427078247, -0.45216336846351624, -0.7063891887664795, 0.5012568235397339, 0.9692360758781433, -1.1018295288085938, 1.8175091743469238, -0.3000926673412323, -1.456363320350647, 0.595673143863678, 0.4378963112831116, -1.2024476528167725, 2.088770866394043, -0.2836858332157135, -0.6080060601234436, -0.2445567101240158, -1.0845322608947754, -0.586142897605896, -0.9142429232597351, -0.8594530820846558, -0.09471195936203003, 0.8208606243133545, 0.3235644996166229, 1.8570079803466797, -0.7651252746582031, 0.8078776597976685, 1.5078305006027222, 0.19533739984035492, 0.16566947102546692, 1.0618884563446045, 0.6132803559303284, -1.1036200523376465, 1.1591671705245972, 0.7733182311058044, -0.01761707291007042, -0.5574578046798706, 0.8070791363716125, 0.9421096444129944, 0.27993717789649963, -1.44913911819458, -1.1896315813064575, -0.2006019651889801, 0.9474869966506958, -0.4636107385158539, -1.804224967956543, 1.1059321165084839, -0.20055623352527618, 0.6461268067359924, -1.3576982021331787, -0.7958219051361084, -0.09871990978717804, 1.2901535034179688, 0.7353525161743164, 0.8140187859535217, -0.71016526222229, 0.964087724685669, 0.904036283493042, 0.4906921088695526, 0.11775032430887222, -0.6146752834320068, -2.103083610534668, -1.2069332599639893, -0.8865183591842651, -0.7731547951698303, -0.2782588303089142, 0.06543031334877014, 0.48490920662879944, -0.0750449076294899, -1.0164371728897095, -0.3779336214065552, -1.1517170667648315, 0.4554528295993805, 1.6082863807678223, -0.3977649211883545, -0.7620385885238647, 1.9934779405593872, -0.34589067101478577, -1.2637490034103394, 0.833116352558136, -1.4816415309906006, 0.22369347512722015, -1.1303507089614868, -0.9148721098899841], [-0.8190230131149292, -1.0554970502853394, -0.3654344379901886, 0.4640790820121765, 0.11882288753986359, 0.822357714176178, -0.10463724285364151, -0.3588089346885681, -1.3252882957458496, -0.11956587433815002, 0.5168290734291077, 2.220036745071411, 0.9422655701637268, -0.2452445775270462, -1.6173326969146729, -1.4482598304748535, 2.16727352142334, 1.0597422122955322, -1.496964454650879, -0.2991664707660675, -0.726933479309082, -1.4175504446029663, 1.1363922357559204, -1.037293791770935, -0.49546101689338684, -0.9787854552268982, 0.30879461765289307, 1.2143549919128418, -0.36425596475601196, 0.3747420907020569, 1.1931731700897217, 0.20770756900310516, -0.46706682443618774, 0.8668103218078613, -0.08903251588344574, 1.3682329654693604, -0.19638466835021973, 0.4449918866157532, -0.61027592420578, 0.5279051661491394, -2.321402072906494, -1.1524004936218262, 0.04581936076283455, -0.5447776913642883, 0.23089013993740082, -1.9452402591705322, -0.04082907736301422, -1.0861012935638428, 1.5104107856750488, -0.17574192583560944, -0.37092316150665283, -0.2941683828830719, 0.029620440676808357, -0.5610527396202087, -0.5086666941642761, -0.4395257532596588, 1.2940696477890015, 1.5653538703918457, -0.09679463505744934, -0.9757251739501953, 0.16349272429943085, -1.4669502973556519, 0.7456310391426086, 1.6889827251434326, -0.8671641945838928, 0.9935098886489868, 0.07605206221342087, -2.4431159496307373, 1.5070699453353882, -0.17711852490901947, -1.311040997505188, 0.8350125551223755, -0.940619945526123, 0.19724924862384796, -1.1491739749908447, -1.5531128644943237, 1.183255910873413, -0.3763028383255005, -1.1236478090286255, 0.08793888986110687, 0.1562477946281433, 1.9392541646957397, -1.7459871768951416, -0.9936801195144653, -1.3087092638015747, -0.14007093012332916, -1.2359700202941895, -1.7529138326644897, 0.6536177396774292, 0.7783889174461365, -0.3258858025074005, 0.9876578450202942, 1.0397840738296509, 1.0795892477035522, -0.12843239307403564, 0.7092807292938232, 0.1941090226173401, -0.7151589393615723, 1.1410038471221924, 0.545892059803009, 0.053440939635038376, 1.1477636098861694, 0.7873377799987793, -1.6622291803359985, 1.8443477153778076, 0.48043492436408997, -0.9413361549377441, 0.6628859043121338, 1.2370201349258423, 1.3297650814056396, -0.0366465300321579, -0.9535700082778931, 0.774307131767273, -1.0213544368743896, 1.944916009902954, 0.10895305871963501, 0.08969536423683167, 1.5696887969970703, 0.2506236135959625, 1.05884850025177, 0.9722610712051392, -0.8818175196647644, -0.9519912600517273, -0.5152143836021423, -0.40291374921798706, 0.5806294679641724, -0.6553013920783997, 0.297658234834671, -0.839303731918335, -1.015199065208435, -0.32923150062561035, -1.0059475898742676, -0.4700559973716736, 1.074486255645752, -0.1362500786781311, -0.011510960757732391, -1.658889889717102, 0.9034276604652405, 1.1531447172164917, -0.48852118849754333, -0.5982648134231567, -1.3157038688659668, 0.47963130474090576, 0.13986948132514954, 1.948809266090393, -0.8856541514396667, 1.546265959739685, -0.4015814960002899, 0.3962622284889221, -1.232211947441101, 0.7633594274520874, -1.4458616971969604, 0.6025040149688721, 1.8188040256500244, -0.21431642770767212, 0.272257924079895, 0.30935439467430115, 2.2955708503723145, 0.6319195032119751, -1.0748225450515747, -0.49978160858154297, 0.3539744019508362, 0.23928259313106537, 2.4390032291412354, -0.9957098364830017, -2.477294921875, 2.3045711517333984, -1.0146996974945068, -0.6399741172790527, 1.1514836549758911, -2.1519882678985596, 0.29043877124786377, 1.6953239440917969, -0.9168938398361206, -0.3630489408969879, -0.12686651945114136, 0.16064953804016113, -0.7003269791603088, 0.4178427457809448, -0.35713711380958557, 0.7989877462387085, -0.4476960003376007, 0.25572583079338074, -0.5056205987930298, -1.0467286109924316, -1.0388818979263306, 0.9727668166160583, 0.8535887598991394, 0.2015758901834488, 0.023498082533478737, 0.0900801420211792, -1.297214388847351, 1.6363250017166138, 0.03495808318257332, -0.7209782004356384, -0.005658434238284826, -0.47029295563697815, 0.5563846826553345, 0.9170575737953186, 0.7303918600082397, -0.18928879499435425, -1.3033138513565063, -0.752677321434021, 1.0299911499023438, 1.143640398979187, -1.316851258277893, 1.3834388256072998, 0.15036724507808685, 1.7702537775039673, 0.38149943947792053, -1.7225701808929443, -0.8794714212417603, -0.5170590281486511, -0.49282902479171753, 1.032431721687317, 0.42413297295570374, -0.8367280960083008, 0.8217350840568542, -1.510099172592163, -1.5235956907272339, 0.048204440623521805, 0.6871182918548584, 0.08906174451112747, -0.22932468354701996, -0.28257954120635986, 0.25261223316192627, 1.1071348190307617, 0.6890696883201599, 1.7702819108963013, -0.5914818644523621, 0.3993707001209259, -0.09122031182050705, -0.3975785970687866, 1.4245716333389282, 0.16182181239128113, 0.29434484243392944, -0.24977129697799683, -0.1203446239233017, -0.8064985275268555, -0.9395362734794617, -1.6997623443603516, 0.018470538780093193, 0.17318886518478394, -0.017662083730101585, 1.3377289772033691, 0.44557371735572815, -1.0343306064605713, 0.7294374704360962, 0.4966854751110077, -0.8559786081314087, -0.12703701853752136, 0.25290465354919434, -0.7060416340827942, -0.5915441513061523, -1.5669795274734497, 1.1614959239959717, 0.972559928894043, 0.9305930137634277, -0.7298616170883179, -0.698422372341156, -0.8708133101463318, 1.224082350730896, 0.7554462552070618, -0.41373172402381897, -1.1091444492340088, -1.3316043615341187, 0.9176675081253052, -0.3988036811351776, -1.1641548871994019, 2.8633344173431396, -0.9302074313163757, -1.119258165359497, 0.9899275302886963, -0.6104116439819336, 0.852218747138977, 0.28519904613494873, -0.8173008561134338, 0.04799655079841614, -0.08402569591999054, -0.7157502174377441, 0.009939712472259998, 1.1503663063049316, 1.33921217918396, 1.1018956899642944, 0.04059448465704918, -0.41497498750686646, 1.409695029258728, 1.42622709274292, 0.8324249982833862, 0.2809741497039795, 0.37044140696525574, -0.21312883496284485, -1.4022258520126343, -0.04786396399140358, 1.1481273174285889, -1.2260679006576538, -1.3633769750595093, -1.320532202720642, 0.5493777990341187, 0.5121846795082092], [-1.270944595336914, 1.9974408149719238, -0.3077159821987152, 0.039302632212638855, -0.6732552647590637, -0.3753603994846344, 1.1670687198638916, -0.41533786058425903, -0.5740531086921692, -0.05606039986014366, 1.3258802890777588, -0.7897728681564331, 1.1156747341156006, 1.0934027433395386, 0.07802126556634903, 0.1265653371810913, 1.4526610374450684, -1.7374833822250366, -0.8938819169998169, -0.11090580374002457, -0.9417317509651184, -0.4477216899394989, -0.9468913674354553, -0.4250444173812866, -0.9014924168586731, -0.18778665363788605, -2.205594539642334, 0.4215468466281891, -0.34198495745658875, -1.3322278261184692, -0.2973159849643707, 0.8963291645050049, -0.7335911989212036, 0.9583007097244263, 0.06585841625928879, 0.1385212540626526, 2.1908347606658936, 0.5101358890533447, 0.03893086686730385, -0.872994065284729, 1.8255302906036377, -2.0931191444396973, 0.8508524894714355, 1.1588553190231323, 0.7359660267829895, -0.4224552512168884, -0.8483334183692932, 0.46711304783821106, -0.1974172294139862, -0.5495261549949646, -1.1914583444595337, 0.22767752408981323, 0.04573069140315056, -1.1798958778381348, 0.31778624653816223, -1.3702011108398438, 0.767123281955719, 0.11716143786907196, 0.06614328175783157, 0.03838673233985901, -0.2696983218193054, -0.9468418955802917, -0.3782530426979065, -1.8188555240631104, 0.654570460319519, 0.1369943469762802, 1.0259424448013306, -0.47286197543144226, -0.5200372338294983, -0.2795904874801636, -0.3728393614292145, -0.8286998867988586, -0.7935704588890076, -0.38709884881973267, -0.6509329676628113, 0.6076589226722717, -0.6448701620101929, 0.5517404675483704, -1.4903367757797241, -0.5084393620491028, 0.4410061538219452, -0.9288166761398315, 0.4961414635181427, 0.7891336679458618, 0.37756916880607605, -0.4660010039806366, 0.10883963108062744, 0.12338770925998688, 1.1535358428955078, 2.1741089820861816, 0.27535614371299744, -1.243565559387207, 1.9798351526260376, -1.177363395690918, 0.7158942222595215, 0.22276665270328522, 0.6517204642295837, -0.3554637134075165, -2.1807539463043213, 1.380001187324524, -2.3987743854522705, -0.8800976276397705, 2.1478214263916016, -0.5411331653594971, 1.5957214832305908, 0.37960347533226013, 0.8951919078826904, 1.3163082599639893, 0.40583938360214233, 0.1906166970729828, 0.8975778818130493, 1.2710740566253662, -0.39036160707473755, -1.8040615320205688, -0.3421907126903534, 0.29615604877471924, 0.7677291631698608, 0.5771229863166809, 1.321303367614746, 1.0069648027420044, -0.1572336107492447, -1.529724359512329, 0.023322081193327904, 0.0699705109000206, -0.21929937601089478, -1.1052377223968506, 1.057323932647705, 0.27135640382766724, 1.1432842016220093, -1.46440589427948, -1.2290563583374023, -1.073559284210205, 0.3353971242904663, 0.8847935199737549, -1.2630828619003296, 0.9474660754203796, 2.1686246395111084, -0.8263785243034363, -0.6297470927238464, -1.8872437477111816, 0.12137731164693832, 2.035743474960327, -0.19222739338874817, -0.30373066663742065, -0.8361579775810242, -1.0925368070602417, 0.5619567036628723, -1.770437240600586, -0.19309750199317932, -0.5314236879348755, -0.8893141746520996, 0.6663979291915894, -0.2704378664493561, -0.5611340403556824, -1.0054104328155518, 1.2378504276275635, -0.28776490688323975, 2.1063332557678223, -0.48036888241767883, -0.13493268191814423, -0.1280762255191803, -1.3259769678115845, -0.42706218361854553, 0.6696451902389526, 1.7846264839172363, 1.9346275329589844, -1.0123659372329712, 0.47222140431404114, -0.41518208384513855, -0.46552902460098267, 1.791119933128357, 3.080571174621582, -2.4535255432128906, 1.6651196479797363, -0.6931179761886597, -1.047745943069458, 0.41806328296661377, -1.2893096208572388, 0.8514900803565979, -0.002600572071969509, -0.8247063159942627, 1.1977001428604126, -0.9855667948722839, -0.018434740602970123, -0.03509395942091942, -1.9061459302902222, -0.25315195322036743, -1.5959323644638062, -0.5442150235176086, 0.9660255312919617, -0.8064410090446472, -1.4188679456710815, -0.4360949993133545, 1.7386246919631958, 1.1256492137908936, 1.3766484260559082, 0.8685337901115417, 0.3430674970149994, 2.6232388019561768, 0.8027907609939575, -1.029004693031311, -0.8267642855644226, -0.7129130363464355, -0.6075329780578613, 0.2620531916618347, 1.3415827751159668, 0.39316752552986145, -0.7593154311180115, 1.4553742408752441, 2.3502719402313232, -0.3230961561203003, 0.7917558550834656, 0.3545536398887634, 1.3845674991607666, -0.23935860395431519, 0.00034792840597219765, 0.22590991854667664, -0.6948363780975342, -1.0578556060791016, -0.767673671245575, -1.9436031579971313, 0.7943568229675293, 1.4453160762786865, -1.4084736108779907, 0.23507583141326904, -1.6144942045211792, -0.6763424873352051, 0.5265777707099915, 1.081264853477478, -0.6885616183280945, 1.1328151226043701, -0.43151000142097473, 1.0345170497894287, -0.14385131001472473, -0.46359261870384216, -0.2356116622686386, 0.019940702244639397, -0.14114736020565033, 0.21318486332893372, 0.9284964203834534, -0.5265878438949585, -0.06697455793619156, -0.3105989992618561, -1.2434288263320923, 1.049185872077942, 1.0247867107391357, 0.825560986995697, -1.4007353782653809, -3.2795538902282715, 0.8723339438438416, 1.0273312330245972, 0.21669571101665497, -0.32181209325790405, 0.37833160161972046, -0.8140592575073242, 0.22224801778793335, -0.41659557819366455, -0.7920688390731812, -0.3182099163532257, 0.058044638484716415, -0.5433995127677917, -1.1131609678268433, -0.561450719833374, 2.590486526489258, 1.2532931566238403, -2.3020336627960205, 0.4529406428337097, 0.4975099265575409, -0.29577499628067017, -0.05651621147990227, -0.75531405210495, -0.33645591139793396, 0.09614954888820648, 0.9968374371528625, -0.4542633295059204, -1.4025170803070068, -0.25966542959213257, 0.49533629417419434, 1.1120038032531738, 0.1698974072933197, -1.0561237335205078, -0.6741892695426941, 0.35971033573150635, -1.348495602607727, -0.5579280257225037, 0.910540759563446, 0.5619369745254517, -1.0364224910736084, -0.7771948575973511, -0.6090967059135437, 0.8785258531570435, -0.5752084851264954, -0.448911190032959, 0.40480735898017883, -0.2913723289966583, 0.636454164981842, -0.04730748385190964, -0.37419620156288147, -0.23929305374622345, -0.057344309985637665], [1.7974621057510376, 0.4645819365978241, -0.7671826481819153, -1.510614037513733, -0.24966992437839508, 0.5876910090446472, -1.4565287828445435, 1.3733768463134766, -0.15876558423042297, 1.2617844343185425, 0.06093256548047066, -1.1338824033737183, -0.18911288678646088, -1.2440890073776245, 1.378198504447937, -0.6740469932556152, 0.5877293944358826, -1.3467960357666016, 1.992627501487732, 0.5312584042549133, -0.6358163356781006, 0.40092864632606506, 1.3756444454193115, 1.5413507223129272, -0.6994014978408813, 0.19266903400421143, 0.042533595114946365, -1.5221301317214966, -0.2807827293872833, -1.1855125427246094, -0.449306845664978, -0.719316303730011, -2.333760976791382, 0.9917165040969849, -0.6142985224723816, 0.2779252231121063, 0.2160293310880661, 2.320523262023926, 1.2157437801361084, 0.036591850221157074, -0.04698099195957184, 1.8584585189819336, -0.4537898898124695, 1.782405972480774, -1.613681435585022, 1.258052110671997, 0.237274631857872, -0.2627154588699341, -1.1733437776565552, 0.18504156172275543, -0.45721474289894104, 0.5539996027946472, 0.9041614532470703, 0.24006006121635437, -0.13861490786075592, 1.8908277750015259, 1.3900706768035889, -2.4594624042510986, 0.020208902657032013, 0.14000998437404633, -0.9622758030891418, 0.16695326566696167, -1.1207540035247803, -0.18510152399539948, 2.1438510417938232, 0.1277860850095749, 0.3311384618282318, -1.0720667839050293, -0.258796364068985, 1.5693830251693726, -0.22602435946464539, 2.346991777420044, -2.543715000152588, 0.31735485792160034, 0.3015573024749756, -1.2566691637039185, 1.3745408058166504, 0.511150062084198, -1.1504937410354614, 0.8915197849273682, -0.19608402252197266, -0.22075489163398743, 0.7974014282226562, 0.9145798683166504, 0.28195732831954956, 0.7415144443511963, 1.0953162908554077, -2.6494393348693848, 1.4646587371826172, -0.5587844252586365, -1.3778010606765747, 0.9716154932975769, 0.4070129990577698, -1.7571334838867188, -0.4110955595970154, 1.336855411529541, -1.0599154233932495, 0.23848673701286316, -1.5594393014907837, 0.9262768030166626, 1.9683724641799927, -0.41247719526290894, -2.069674015045166, -0.9567769765853882, 0.22573164105415344, -0.36123645305633545, 0.5498935580253601, 0.7326725125312805, -0.914699375629425, -0.8387503027915955, -0.2822321951389313, 0.21006497740745544, -2.4617273807525635, 1.5764045715332031, 0.56792813539505, -0.41481518745422363, -0.3112475275993347, 0.12594354152679443, -0.022686069831252098, -1.1568714380264282, -0.336197167634964, -1.937078833580017, 0.8094640374183655, 0.9276690483093262, 0.42134755849838257, 0.14308588206768036, -0.4337182343006134, -1.09064781665802, -1.1392298936843872, -1.1486529111862183, 0.3731226325035095, 0.9402089715003967, 0.2519010901451111, -0.7016859650611877, -0.5554264783859253, 0.16988196969032288, -0.08500790596008301, -1.135428547859192, -0.14747239649295807, -0.9787161946296692, -0.9345881938934326, -0.5296493172645569, -0.6528245210647583, -1.7588138580322266, 0.38109737634658813, 0.19538316130638123, 0.9558898210525513, -1.799463152885437, -0.14005570113658905, 1.2154154777526855, 0.7250133752822876, -0.6642006635665894, -0.12640182673931122, 0.13096918165683746, 2.4295880794525146, -0.392799437046051, 0.6897060871124268, 1.1393071413040161, -0.982692539691925, -0.6964243054389954, 1.683416485786438, -0.37130680680274963, -1.0785858631134033, 0.31769198179244995, -1.1392158269882202, -1.771163821220398, 0.9309945106506348, 1.1567775011062622, 1.0385421514511108, -1.1826261281967163, 1.3307331800460815, 0.8751331567764282, 1.2858473062515259, 0.10092546045780182, 1.1533559560775757, 0.7863492369651794, -1.3115794658660889, -1.135602593421936, 0.5429049134254456, -1.7097455263137817, 2.0012338161468506, -1.079558253288269, -0.9768334031105042, -0.9822267889976501, 0.2797040641307831, 1.5978763103485107, 0.26243457198143005, -2.0193939208984375, 0.053623367100954056, 0.0663483738899231, 2.1360185146331787, 0.9773586392402649, 0.2755405902862549, -1.7044659852981567, -0.13285526633262634, 1.74537992477417, 0.4581177234649658, -0.12904982268810272, 0.9123591184616089, 1.2002184391021729, -0.189626082777977, -0.5848904252052307, 0.18233735859394073, 0.547942578792572, -0.512161374092102, -1.1472828388214111, 0.8593170642852783, 1.002353549003601, 0.17235398292541504, 0.0605255663394928, 1.4368189573287964, -0.4265062212944031, -0.4146837890148163, -1.2788684368133545, -1.2302227020263672, 1.190280556678772, 0.7964588403701782, -1.3159395456314087, 0.5966787338256836, 2.0478925704956055, 0.06761530041694641, -1.7612789869308472, 0.7667379379272461, 0.4836050570011139, 0.4109661281108856, -0.858074963092804, 0.8596064448356628, -1.607168436050415, -0.7924936413764954, -0.8770323395729065, -0.46542423963546753, 1.7000436782836914, -0.551914632320404, -0.35513341426849365, -0.030822966247797012, -0.2237865924835205, -2.1050472259521484, -0.8222700357437134, -1.2587847709655762, 0.61381596326828, -0.1255623996257782, -1.1227834224700928, -1.6817090511322021, 0.18879248201847076, 1.054590106010437, -2.559800386428833, -1.8289977312088013, -0.3022051751613617, 0.5706446170806885, 0.5896993279457092, 0.23344077169895172, -0.8879457712173462, 0.036252524703741074, 0.15480154752731323, -1.4745237827301025, -0.07087879627943039, 1.0973761081695557, 1.6984041929244995, 2.1032040119171143, -1.8214808702468872, 0.21563291549682617, -0.9221317172050476, -0.3292381465435028, -2.3362300395965576, -0.24723167717456818, -0.27591636776924133, -1.5441439151763916, -0.7235389947891235, 0.7027772665023804, -1.352234959602356, 0.4262513816356659, -0.14933142066001892, -0.7659063339233398, -0.7002494931221008, 0.31443995237350464, -0.5852208137512207, -0.0097044063732028, 0.764245867729187, -0.08307964354753494, 0.6590297222137451, -0.40420031547546387, 0.5120287537574768, -0.600048840045929, -1.2168796062469482, 0.5000327229499817, 0.8901084661483765, 0.4373320937156677, -0.07660169154405594, -0.9769161939620972, -0.5412769913673401, -0.5029990077018738, 1.557403564453125, -0.8125903010368347, 0.6414544582366943, -1.1694868803024292, 2.289818286895752, -0.16998110711574554, -0.274698942899704, 0.5333188772201538, 0.707319974899292], [0.12787966430187225, -0.8974170684814453, -1.5721938610076904, -0.1575377732515335, 1.8879810571670532, 0.583828330039978, -0.9586374163627625, 0.28120967745780945, 0.7781072854995728, 1.5000065565109253, -0.45555102825164795, -0.8915072679519653, 0.9534273743629456, 0.8825953602790833, 0.6492288112640381, 0.2893228828907013, 1.624030590057373, 1.0348953008651733, 1.106163501739502, 1.1289186477661133, 0.23053976893424988, 1.5454139709472656, 0.7590567469596863, -0.23082609474658966, 0.38942548632621765, -1.9183707237243652, 0.44119080901145935, 0.44933533668518066, -1.2766647338867188, 0.8455107808113098, -1.3261123895645142, -1.240518569946289, -0.6739295125007629, 0.4076932966709137, -1.078157901763916, -0.5233913660049438, 1.3484989404678345, -1.5009499788284302, 0.421530157327652, 1.0209940671920776, -0.029626844450831413, -1.637821912765503, -0.7866348624229431, -1.2007606029510498, -0.5659030675888062, -0.49524399638175964, 1.8442355394363403, -1.3542078733444214, 2.3418827056884766, -0.02821251004934311, 0.4995872378349304, 0.2022707760334015, 0.5644466876983643, -0.3392387628555298, 1.5928295850753784, -1.6814876794815063, 0.3878914415836334, 0.03858289122581482, -0.8691949844360352, -0.02674339897930622, 0.10000511258840561, 0.9997990727424622, 0.5014744400978088, -0.45842012763023376, 0.21991613507270813, 1.6155080795288086, -0.7416672706604004, 0.17204904556274414, 0.5306733846664429, 0.3802259564399719, 0.36963093280792236, 0.25905150175094604, -0.6670873761177063, -0.8298366069793701, 0.19780507683753967, 0.6137118339538574, -0.8540666699409485, 0.7078593373298645, -0.2263287752866745, 0.8256216645240784, 0.14518816769123077, -0.48081398010253906, -2.467871904373169, 0.6751143932342529, -1.2882790565490723, -0.22632762789726257, 1.4871330261230469, 0.11580348014831543, -0.429486483335495, 0.3332245647907257, 0.16569368541240692, 1.7066336870193481, -0.42571744322776794, -1.1497385501861572, 0.22686195373535156, 0.44923511147499084, -1.5048792362213135, -0.7627610564231873, -0.49881452322006226, 2.3875014781951904, 0.9119845032691956, -0.15047866106033325, -0.7949928045272827, -0.07957574725151062, -0.7451903820037842, -0.009763217531144619, -1.2318918704986572, -0.7521374821662903, 2.3858261108398438, 0.0838836207985878, 2.264763355255127, -0.36033201217651367, 0.02698693238198757, -0.6329845190048218, 1.0393449068069458, 0.4184207618236542, -1.4000645875930786, -0.3810746669769287, -1.2313482761383057, -0.9643949866294861, -0.2526423931121826, -0.9595147371292114, -1.8859034776687622, -0.7442104816436768, 0.6680537462234497, 1.3249874114990234, 1.7009121179580688, -0.47563591599464417, -1.4467222690582275, -0.0204722099006176, 0.683571457862854, 1.1226409673690796, 1.592939019203186, -0.2089618593454361, -1.3280153274536133, -0.16335679590702057, -1.9776899814605713, 0.9089592099189758, -0.47812485694885254, -0.16873855888843536, 0.8554713129997253, 1.1335424184799194, 0.030798234045505524, -0.0928620994091034, 0.08447019755840302, -0.4390796720981598, 0.1878812164068222, -0.9656656980514526, -0.9076263904571533, -0.9202338457107544, -0.9387664198875427, 0.6307281851768494, -0.6741605997085571, 0.4418061673641205, -0.4901639521121979, 1.3218339681625366, -1.6220104694366455, -1.2787379026412964, -1.8223304748535156, 1.6467100381851196, -0.8547009229660034, 0.7158415913581848, 0.592595636844635, -1.8248673677444458, 1.0025877952575684, -0.3384384512901306, -0.243878573179245, 0.07979216426610947, 0.7054104208946228, 0.9768045544624329, -0.18558426201343536, 0.24371355772018433, 2.037463903427124, 0.23321878910064697, -0.2793959379196167, -0.7713099718093872, -1.1311873197555542, 0.24294358491897583, 0.3897162973880768, 2.1780810356140137, 0.23057083785533905, -0.19807562232017517, 1.3417922258377075, -1.3689297437667847, -0.1432255655527115, 1.252487301826477, -1.0884184837341309, -0.10543297976255417, -0.8204114437103271, -0.04759783297777176, 1.261357069015503, -1.890169382095337, -1.4129103422164917, 0.319259375333786, 0.2165479063987732, 0.2211993783712387, 0.005264502950012684, -1.019014835357666, -0.27524659037590027, 1.0727037191390991, -0.4044063985347748, 0.3594288229942322, -0.2690421938896179, -0.5019702911376953, -0.458429753780365, 1.9185069799423218, -1.7916967868804932, -0.29161983728408813, -0.41915395855903625, -1.408042311668396, 0.4294222891330719, -0.4629356861114502, 1.1500608921051025, -2.6192033290863037, -0.02405562251806259, 1.8266857862472534, 0.345409631729126, 0.6058886647224426, 0.22086676955223083, 0.7838223576545715, -1.769355297088623, 1.0705941915512085, -0.09822095930576324, 1.267587423324585, -1.6320246458053589, -0.152817040681839, 1.7183725833892822, -0.26586249470710754, -1.2088199853897095, 0.6851575970649719, 0.5693206787109375, 0.2723313271999359, -0.20491182804107666, -0.06276394426822662, -0.4812074303627014, 0.346099317073822, -0.047026168555021286, -0.03804532438516617, -1.1655205488204956, 0.14362147450447083, 0.7442244291305542, 0.5360622406005859, -1.49591863155365, -0.8468591570854187, 1.4356149435043335, 1.3255304098129272, -1.160487413406372, -0.1527673751115799, 1.1532678604125977, 0.7393761277198792, 0.3034118115901947, -0.06515881419181824, -0.4843762516975403, -0.04923461377620697, 1.0714086294174194, 1.2207344770431519, -0.6448222994804382, 0.16229234635829926, 0.8018472194671631, 1.3064357042312622, 1.015212059020996, -0.8717243671417236, -0.44053465127944946, 1.0550514459609985, -1.8201202154159546, 0.8676977753639221, 1.8513096570968628, -0.9599626660346985, 0.30118197202682495, 1.2864787578582764, -0.14767985045909882, 0.3655781149864197, -2.0034780502319336, -1.0558958053588867, 0.3465489149093628, 0.05208820104598999, 1.082125186920166, 0.6271353363990784, -0.2394426316022873, -0.9834097027778625, 0.7289991974830627, 1.2683180570602417, -0.6157314777374268, -2.0859978199005127, 0.5052875876426697, -1.6621339321136475, -0.25996533036231995, 0.24526390433311462, 2.8206772804260254, 0.7850461602210999, 0.2485663741827011, -1.124222993850708, -0.018086420372128487, -0.18869154155254364, -1.7228659391403198, -1.139849066734314, 1.7903220653533936, 0.6695041656494141, -0.1429712325334549, -1.6511859893798828], [-0.4494044780731201, 1.1239464282989502, 0.8954454660415649, -0.46738457679748535, -0.6623297929763794, -0.3139580190181732, 0.08366046845912933, 2.309022903442383, 0.2718805968761444, -0.06805793941020966, 1.8528721332550049, 0.20139773190021515, 0.9779273271560669, 2.1469218730926514, -0.26746609807014465, -1.1668483018875122, 1.6611974239349365, 1.2348333597183228, -0.12085667252540588, 0.6881262063980103, 0.3037962019443512, 0.541098415851593, 1.1906989812850952, 1.8400588035583496, -1.1632903814315796, 1.0628608465194702, -1.4764567613601685, 1.004658818244934, -0.014438651502132416, 0.15607579052448273, -1.516842246055603, -0.8402930498123169, 0.7520660758018494, 0.7583260536193848, 1.0851532220840454, -0.8428635001182556, 2.0203137397766113, 0.16423022747039795, 0.5626379251480103, -0.3120306134223938, -1.7027055025100708, 0.9340363144874573, -1.1441149711608887, -0.6758456826210022, 1.6317702531814575, -0.39753803610801697, -0.6825701594352722, 0.05994679778814316, 0.07130695134401321, -0.8796228766441345, 0.07321648299694061, 2.1755237579345703, 1.8199409246444702, -0.07430823147296906, -1.0328925848007202, 0.3949851393699646, 1.8751667737960815, 0.33185020089149475, 0.5878582000732422, 0.5698775053024292, 0.1391368955373764, 2.0222957134246826, -2.3046863079071045, 0.6656813025474548, -0.18590538203716278, 0.3101147413253784, 0.024400046095252037, -0.4081230163574219, 0.5323882102966309, 0.26868194341659546, 1.2288970947265625, -0.29491275548934937, -1.5721025466918945, -0.522294819355011, -0.0007198081002570689, -0.19782397150993347, -2.040804624557495, -1.9691990613937378, 0.6726105809211731, 0.39732393622398376, 0.1774972677230835, 0.5540910363197327, 0.26841434836387634, -2.0511085987091064, 1.2552133798599243, -0.4046059250831604, 0.1919664740562439, -0.07401400804519653, -0.9980823993682861, 0.3637244999408722, 0.8731896877288818, 0.16013935208320618, -2.0000648498535156, -0.4070313274860382, 0.6840401291847229, 0.46994704008102417, -1.312058687210083, 0.009415913373231888, -0.41071146726608276, 1.093849539756775, -0.36808621883392334, 0.19494999945163727, 0.22558821737766266, -2.0642619132995605, 1.9867322444915771, 1.3038338422775269, 0.18977075815200806, -0.680048942565918, 0.20390667021274567, 1.0452457666397095, 2.9842896461486816, 1.7148176431655884, -0.3430198132991791, 1.529068112373352, 1.1441541910171509, -0.2370460331439972, 1.2506275177001953, 1.0053104162216187, -0.5954004526138306, 1.4643350839614868, 0.04697933793067932, 0.14474685490131378, -0.5772007703781128, -0.6435911059379578, -0.06335900723934174, 1.132191777229309, 0.47971343994140625, 1.5083894729614258, -0.3733871877193451, -0.07332759350538254, -0.37003055214881897, -0.43173453211784363, -1.311948299407959, -0.16722778975963593, 2.459392786026001, -0.3490979075431824, 0.39518865942955017, 1.3724751472473145, -0.14373502135276794, -0.1463056057691574, 2.5323731899261475, 1.029853343963623, 0.17662228643894196, -1.2467010021209717, -0.04598876088857651, 0.6520925164222717, -0.06310320645570755, 0.06315349042415619, 0.9992051720619202, -0.16152240335941315, 0.1658506989479065, -0.11921682953834534, 1.365047574043274, -0.7784196734428406, -0.8871097564697266, -1.2433154582977295, 0.6804215908050537, 2.1670827865600586, 0.8689841032028198, -0.367883563041687, 0.10716482251882553, -0.17803184688091278, 0.6452668309211731, 0.017441267147660255, 0.23009194433689117, 0.5503671169281006, -0.3986779451370239, 1.1065456867218018, -0.30082830786705017, 2.153724193572998, 1.5730116367340088, -0.8225418925285339, -0.6642723679542542, 0.035231977701187134, -0.599122941493988, 1.434998869895935, -0.4589567184448242, 1.4432916641235352, 0.5266220569610596, -0.016884084790945053, -2.2947499752044678, 0.6237938404083252, -1.3010510206222534, -0.08201546967029572, -1.039712905883789, -0.6994898319244385, -0.32696229219436646, -1.0420514345169067, 0.0671825036406517, -0.585755467414856, 0.020404228940606117, -1.122604489326477, 0.8001254796981812, 1.6342819929122925, 0.6922551989555359, 0.5244850516319275, 0.7788370847702026, 0.4548323452472687, -1.3963109254837036, 2.144158363342285, -0.7408652305603027, -2.102015733718872, -0.6695685982704163, -0.4222226142883301, 0.2518652379512787, 0.1509283483028412, 0.13191455602645874, -1.8161641359329224, 0.6881927847862244, -0.6169942617416382, 1.6852754354476929, 1.4851956367492676, 0.7891191244125366, 0.9650477766990662, 0.4234355390071869, -2.3718817234039307, -1.299496054649353, 0.49510321021080017, -0.2996854782104492, 0.9299176335334778, 1.1887925863265991, -0.807308554649353, -0.5303178429603577, -1.5173412561416626, -0.3155510127544403, 1.254724383354187, 1.0519859790802002, 0.13795126974582672, -0.3661052882671356, 1.4146218299865723, 0.047806255519390106, 0.0035605300217866898, 0.19808150827884674, 0.8476561307907104, 0.013063893653452396, -0.07102937996387482, 1.1413459777832031, 0.6435047388076782, 0.04673362895846367, 0.1068030372262001, 0.055609095841646194, 0.028377152979373932, 0.8715705871582031, 1.0901881456375122, 1.0168430805206299, -1.3945634365081787, 0.16303639113903046, -1.900282621383667, 0.699653148651123, -0.4766993224620819, -2.141308546066284, 0.058586690574884415, -0.09472562372684479, 1.685874104499817, -0.5030015110969543, -1.9501205682754517, -1.1153219938278198, 0.09290418028831482, 0.06063311547040939, 0.276877760887146, 0.46852219104766846, -0.724789559841156, -0.9191310405731201, -0.43478837609291077, 0.6064713001251221, 0.7333484888076782, -0.9703256487846375, 0.7164463996887207, -0.42698195576667786, -0.48457175493240356, 0.20028471946716309, -0.39344751834869385, 0.2129380702972412, 0.03975358232855797, -0.33524641394615173, 1.7564669847488403, 0.17038291692733765, -0.46502670645713806, 1.8909693956375122, 0.19547595083713531, -2.1695520877838135, 1.4153209924697876, -0.22329869866371155, 0.15216095745563507, -1.1616066694259644, 0.35734838247299194, -0.6687771677970886, -0.97779381275177, 0.6685526371002197, 0.6787553429603577, 0.649895429611206, 0.017968609929084778, 0.15314370393753052, -0.4159398078918457, -0.1974979043006897, 0.058045223355293274, 0.9305845499038696, -0.17805281281471252, -1.5780740976333618, 0.06138428673148155], [-0.4459892213344574, -0.1250544786453247, -0.39468878507614136, -0.5708667635917664, 0.24690550565719604, 1.609973669052124, -1.6394301652908325, -0.9079628586769104, -1.1444072723388672, -0.15909062325954437, 2.6911251544952393, -0.06685808300971985, -0.9095937013626099, 1.5069315433502197, -0.8615874648094177, -0.17609718441963196, -0.3055693209171295, 1.0709197521209717, 0.9128873944282532, -0.2561650276184082, -0.06726402789354324, -0.03994445502758026, -0.2470196783542633, -2.084634304046631, -0.19580639898777008, -0.6669766902923584, -0.21999049186706543, 0.018651174381375313, 0.594878077507019, 0.6827822327613831, 1.1791372299194336, 0.32328006625175476, 1.2304270267486572, 0.24687953293323517, -0.14253926277160645, 1.1737593412399292, -0.19911466538906097, -0.7979341149330139, -1.1865239143371582, -1.0492812395095825, 0.28694507479667664, -0.16058310866355896, 1.0842232704162598, 1.1570909023284912, -0.6882616877555847, 1.268781304359436, -0.2147865742444992, -0.6434815526008606, -1.2853667736053467, -0.5585610866546631, 0.5814186930656433, -0.3827546536922455, -0.10712921619415283, 0.27356424927711487, 0.8980914354324341, -0.2898878753185272, 0.16007794439792633, 0.18068617582321167, -1.0426877737045288, -0.6530666351318359, 0.41721311211586, 0.48945215344429016, 0.8216785788536072, 2.0580663681030273, 0.43939462304115295, 1.1434229612350464, -1.3388639688491821, 0.9158616065979004, -0.1362379640340805, 0.4111977517604828, 1.717713713645935, -0.3253890872001648, -1.091398000717163, 0.7204797267913818, 1.1690895557403564, 1.1550369262695312, -0.32882270216941833, -1.3895964622497559, -1.5053290128707886, 1.0690107345581055, 1.3973543643951416, 0.42796438932418823, 2.4020814895629883, 1.20822274684906, -0.6578355431556702, 0.3044341504573822, 0.12446283549070358, 0.23878826200962067, -1.720101237297058, 1.5405957698822021, -0.17000353336334229, -1.1857478618621826, 0.660097062587738, -0.24115131795406342, -0.496135413646698, 0.15034160017967224, -0.15547536313533783, 0.3307894468307495, 0.7918732762336731, 0.5985985398292542, -0.34075647592544556, -0.3226671516895294, 0.10292524844408035, 0.6567209362983704, 0.5043033361434937, 1.4999524354934692, 1.3267565965652466, 1.46407151222229, -2.8768389225006104, 0.09884923696517944, 1.82383394241333, -0.9918344616889954, -0.3771612346172333, -1.043630599975586, -0.6957570910453796, -0.7553741335868835, 1.146820068359375, -1.4977774620056152, 0.7182328701019287, 0.20217645168304443, 1.0332752466201782, 0.2415744960308075, -0.25470486283302307, 2.2978620529174805, -0.4917536675930023, -2.651224374771118, 0.7843590378761292, -1.3794370889663696, 1.6749495267868042, 2.746154546737671, 0.14744196832180023, -1.1001465320587158, 0.05073785409331322, 0.9718742966651917, 0.9852414131164551, 0.1483965814113617, -0.8107317090034485, 1.5439854860305786, 1.481472134590149, -0.014891796745359898, -0.304228812456131, -0.611664891242981, -0.6902062296867371, 0.012645668350160122, -0.2573905289173126, 0.15179850161075592, -1.120834469795227, 1.011389970779419, 0.939508318901062, 0.17066054046154022, -0.7606410384178162, 0.21363095939159393, 1.1263631582260132, 0.39798930287361145, -0.4177655577659607, 0.22351375222206116, 0.44058486819267273, 0.9365288615226746, 1.525012493133545, -0.3139599859714508, 0.4860568344593048, 0.8871846795082092, 1.7084962129592896, 1.8077647686004639, 0.3112696707248688, 0.1274481862783432, 0.3988923132419586, -0.9828872084617615, -1.0180765390396118, 2.9285597801208496, 0.7165219187736511, -1.5698952674865723, -1.2497087717056274, 0.3371482789516449, 0.5768864154815674, -1.404360294342041, 0.6431859135627747, -0.16796062886714935, -1.0661418437957764, 0.2764572501182556, -2.1668570041656494, -0.08083811402320862, -0.9452610611915588, -1.3243696689605713, 0.77751624584198, 2.601407766342163, -1.0310922861099243, -0.5659205317497253, 0.5811271071434021, 0.5887534618377686, 0.19863702356815338, 1.8070801496505737, -0.7198851108551025, -0.23033519089221954, 0.005940424278378487, 0.8939895033836365, 0.8000223636627197, 0.21823816001415253, 0.786851167678833, -0.46143749356269836, 0.24310682713985443, 1.3680179119110107, 0.9127354025840759, 0.08413363248109818, -1.454596757888794, 0.596523106098175, 1.0738325119018555, -0.433855265378952, 0.38265201449394226, 0.28718480467796326, 1.5072330236434937, 0.6393944621086121, 0.11453033983707428, -0.12113512307405472, 1.2964580059051514, 0.3376888930797577, -0.35485512018203735, 0.19803224503993988, 0.2833194136619568, 0.13628706336021423, -0.46369895339012146, 0.7572038173675537, 1.5456035137176514, -0.25855013728141785, -1.1149210929870605, -2.431776762008667, 0.8444661498069763, -1.247077226638794, -0.3287647068500519, -0.29377099871635437, 0.4545935392379761, -0.8639171719551086, -1.3910492658615112, 0.5893490314483643, 0.2354137897491455, -0.1446811407804489, -0.030290041118860245, -0.8171995878219604, -0.7235836386680603, 0.3922932744026184, 0.6273026466369629, -0.788114607334137, 1.4196773767471313, -0.03712882101535797, -0.022615427151322365, -0.1570224165916443, 0.19526013731956482, 0.5727704763412476, 1.6487637758255005, -0.07791974395513535, 0.047403484582901, -0.9409890174865723, -0.7047320604324341, 2.2309508323669434, -1.3354324102401733, -0.9266889691352844, 1.1943250894546509, 0.21829107403755188, 0.7982767224311829, 0.2567792534828186, -0.9154268503189087, -1.0493056774139404, -0.4581063389778137, 1.4028676748275757, 1.4428483247756958, -0.9157055020332336, 0.02765215002000332, 0.633805513381958, -0.4973388910293579, -0.2794944643974304, 0.6239700317382812, -1.207745909690857, 0.2724500000476837, -0.7530074715614319, 1.2201470136642456, 0.7865963578224182, 0.28449735045433044, -0.018686098977923393, -0.32803255319595337, -0.4837120473384857, 1.2252769470214844, 0.5735689997673035, 0.4134182631969452, -1.1766250133514404, -0.1524563431739807, 1.1265993118286133, 1.0441253185272217, -1.6245296001434326, -0.8190940022468567, -0.51836097240448, -0.5249816179275513, -0.051787640899419785, 2.0422251224517822, 0.28578755259513855, 0.3388141989707947, -1.3556442260742188, -0.5682132840156555, 1.1874750852584839, -1.6406776905059814, -1.6950739622116089], [0.0007483739755116403, -0.43998509645462036, 1.0605441331863403, -0.1666649729013443, -0.09833845496177673, -3.131443738937378, -0.3969189524650574, -0.11531687527894974, 1.9580705165863037, -0.7247570753097534, 1.4498822689056396, 0.451474666595459, -1.0925796031951904, -0.9061669707298279, -1.3738001585006714, 1.3575295209884644, 1.3399074077606201, -0.9144321084022522, -0.8071321845054626, -0.7670062184333801, 1.970261812210083, 0.07657739520072937, -0.07898637652397156, -1.2407448291778564, -0.29894933104515076, -0.5373387932777405, 0.18727871775627136, -0.3992873728275299, 0.04953298345208168, -0.08700460940599442, 0.667583703994751, 0.2517111003398895, -1.1612259149551392, -0.9382176995277405, 0.9180259704589844, 1.5648384094238281, 1.5471173524856567, 0.19618359208106995, -0.3112008273601532, -0.3624993562698364, -0.8777361512184143, 0.7061097025871277, 1.5747580528259277, 0.9208737015724182, -0.4846889078617096, 0.20211000740528107, 2.59390926361084, 0.9968854188919067, -0.34272661805152893, 0.0372665636241436, -1.0195196866989136, -0.6704550981521606, -0.18032673001289368, 0.10624483227729797, -0.5423927307128906, -0.7241440415382385, 1.154872179031372, -1.0469151735305786, 0.4690503180027008, -0.41937097907066345, 0.471843421459198, 0.7743542790412903, -0.5268761515617371, 0.6635503768920898, 2.1277220249176025, 0.5040847659111023, 0.35072875022888184, -0.5759980082511902, 0.6187519431114197, 0.15243546664714813, -1.6044795513153076, 1.016857624053955, 1.6076006889343262, -1.1119502782821655, 0.17950209975242615, 0.9115449786186218, 0.40693309903144836, -0.75364750623703, -0.8343209624290466, 0.08125877380371094, 0.591376781463623, -0.7618665099143982, 1.6076053380966187, 1.3491439819335938, 2.0048258304595947, -0.4224099814891815, 0.030548419803380966, 0.11764730513095856, -0.6248409152030945, -1.1518932580947876, -0.5751155614852905, 0.8776294589042664, -1.095526933670044, 1.3717195987701416, -0.8322714567184448, -0.42734289169311523, -0.1112334355711937, -0.7101020812988281, -0.6547009348869324, 1.2461882829666138, -1.4901055097579956, 0.5445079803466797, -0.3951966464519501, -0.11186044663190842, 0.26179149746894836, -1.4405591487884521, -0.21024106442928314, 0.5908946394920349, 1.5080461502075195, -0.9517775774002075, -0.7743977904319763, -0.34176039695739746, 0.3543093502521515, -1.0767362117767334, 0.9165261387825012, -0.8327213525772095, -0.5688069462776184, 2.720109462738037, -1.4024382829666138, 0.4676385521888733, 0.014034450985491276, 0.4518740177154541, -0.38468509912490845, -0.8463848829269409, -0.7261309027671814, 0.27883049845695496, 1.3243473768234253, -0.18992207944393158, 0.6096882820129395, -0.4476081132888794, 0.4327590763568878, 0.5448867678642273, -2.089839220046997, 0.554297149181366, -0.43450212478637695, -0.7291098237037659, 0.29668891429901123, 0.662294328212738, -1.399747610092163, 1.2181401252746582, 0.2957342565059662, -1.8448888063430786, 0.1385774463415146, -0.01757396012544632, 0.2786727249622345, 1.0532764196395874, -1.4899579286575317, -1.1839596033096313, 0.24681596457958221, 0.6483036279678345, 1.1051998138427734, -0.8370940685272217, 1.206415057182312, 0.5479846596717834, 1.4080374240875244, 1.598659634590149, -0.4168337285518646, -0.1332017034292221, -0.6811475157737732, -0.12815642356872559, 0.3371308445930481, 0.08253392577171326, -0.8208041787147522, -0.8093134164810181, 0.7804543972015381, 0.08926999568939209, 1.2609786987304688, 0.6092975735664368, -0.5186023116111755, -2.024420976638794, 1.5487641096115112, -0.11552421748638153, -1.6074092388153076, -0.15030480921268463, 0.9146438837051392, -1.3915306329727173, -1.3575860261917114, 0.756841778755188, 0.7994316220283508, -0.6485192179679871, 2.3298327922821045, 0.4261402189731598, 0.07001431286334991, -0.5870102047920227, -0.6246994733810425, -0.27353838086128235, 0.12169928848743439, -1.247023582458496, -0.1899430751800537, -0.46180152893066406, 1.551750898361206, -1.1051594018936157, -1.8057023286819458, -0.9102754592895508, -1.2187689542770386, 0.1701052486896515, -0.6980941891670227, 1.3187298774719238, 0.382609486579895, 1.7343169450759888, -0.3761883080005646, -0.3462400436401367, 0.7160432934761047, -1.5031139850616455, -1.0753949880599976, 0.3645026385784149, -0.7552076578140259, 0.5364590287208557, -2.2061259746551514, -1.3239772319793701, -0.38681137561798096, 0.8486210107803345, 0.9539577960968018, 0.3921141028404236, -0.9129204154014587, 0.7158628106117249, -0.06748499721288681, -0.6924431920051575, -1.2734752893447876, 0.9997873902320862, 0.20199845731258392, 0.2517521381378174, 0.12236246466636658, -0.9440647959709167, -0.35112112760543823, 0.291350394487381, 1.6872615814208984, -0.6988181471824646, 0.6019049286842346, 1.0497961044311523, 2.183562994003296, 0.1333630383014679, 0.5253629684448242, -1.5989751815795898, 0.8249896764755249, -1.3128858804702759, -0.30180323123931885, 1.1572928428649902, 0.6831442713737488, -0.7684599161148071, -1.2022157907485962, 0.5839968323707581, -0.6150808334350586, 0.08740533143281937, -0.4759390652179718, 0.9919515252113342, -2.0103418827056885, -1.3399457931518555, -1.051146388053894, 0.45499128103256226, -1.9505469799041748, 2.6019375324249268, -0.05784529075026512, -0.07494978606700897, 0.006722576916217804, 0.6659344434738159, -0.8064190745353699, -1.170785665512085, -0.3290066719055176, -0.07271837443113327, 0.44874322414398193, 0.3182491362094879, 0.403117299079895, 2.269275188446045, -0.2555951178073883, 0.41938742995262146, 0.5858752131462097, 1.0902122259140015, -1.3625506162643433, 0.27437272667884827, 0.8336706161499023, 1.5179882049560547, 1.264256477355957, -1.5571821928024292, 0.7807310223579407, 1.2278510332107544, 1.4641857147216797, -1.7410449981689453, 1.1188931465148926, 0.327818363904953, -2.064146041870117, -1.713657021522522, 0.6605917811393738, -0.7339949607849121, 0.27414119243621826, 3.1025867462158203, 1.4506757259368896, 1.5345011949539185, -0.8554232716560364, 0.07493628561496735, 0.06380219757556915, 0.7193397879600525, -1.3396422863006592, -1.269926905632019, 0.02511611208319664, -0.5895357728004456, 0.2947039008140564, 0.3327830731868744, 0.6573824286460876, 0.6404312252998352], [-1.4236029386520386, 0.27192792296409607, -0.17902113497257233, 0.7829605937004089, -0.5460106134414673, -0.1132717877626419, 0.814288318157196, 2.1468491554260254, -0.41465240716934204, 1.1116763353347778, 0.4487355649471283, 2.3694796562194824, -0.21812187135219574, -0.47185826301574707, -1.5811583995819092, -1.754337191581726, 0.9659175276756287, 0.4774895906448364, -0.33091235160827637, 0.03830917552113533, -0.513638973236084, -0.8051328063011169, -2.380253791809082, -0.0525989904999733, 0.13868960738182068, -1.4911774396896362, 1.5053685903549194, 0.5071942806243896, 1.0806678533554077, -2.9814462661743164, -0.2401210218667984, -0.3508690893650055, -1.493434190750122, -0.5595189929008484, 1.2419087886810303, -0.0639391615986824, -0.5748775601387024, -0.5738074779510498, 0.09267164766788483, -1.9394043684005737, -0.09544013440608978, -0.7244421243667603, 0.6074349284172058, 0.5669960379600525, 0.9301307797431946, 0.6811097264289856, 0.3971083462238312, 0.8244602084159851, -0.21556229889392853, -1.3680648803710938, -2.1397693157196045, -1.8768290281295776, -1.657970905303955, 0.6550120711326599, -0.4033551514148712, 0.17152844369411469, 0.190006285905838, 0.28465867042541504, -0.6578860282897949, 0.35870254039764404, -1.3223425149917603, 1.753685712814331, -2.130753517150879, -1.5801787376403809, 0.3230167329311371, 0.8429174423217773, -0.633139967918396, -1.568501591682434, -0.4722733795642853, 1.2210187911987305, 0.4757649004459381, -0.710409939289093, -0.1686045229434967, -1.6779989004135132, 1.108302354812622, 1.461205244064331, 0.5305063724517822, 0.012638538144528866, -0.9890498518943787, 2.4610719680786133, -0.6308083534240723, -0.8177754282951355, 0.5182980895042419, -0.16792263090610504, 0.7890285849571228, -0.5023918151855469, 1.0705190896987915, 1.9036638736724854, 0.6241539716720581, 0.2552057206630707, -0.6282598972320557, -0.26945897936820984, -0.2764289379119873, -1.2774128913879395, 0.8901283740997314, 0.0037884851917624474, -1.8735467195510864, -0.20907780528068542, -0.4482758939266205, 1.0084083080291748, 0.23774302005767822, -1.1008366346359253, -1.0181232690811157, 0.12598258256912231, -0.38183748722076416, -0.5203655362129211, -0.1992722749710083, -0.6570732593536377, -1.9210788011550903, 0.24391737580299377, -1.025754690170288, 1.5817477703094482, -1.377589464187622, -1.7339420318603516, 1.8829679489135742, 0.32020053267478943, 0.9491705298423767, 0.631588876247406, 0.1892356425523758, -0.8717358112335205, -0.29486003518104553, 0.21999850869178772, -1.4391108751296997, 0.753858745098114, 0.9444038271903992, 0.013431993313133717, 0.9123220443725586, -1.1146185398101807, 1.2015275955200195, 0.29521632194519043, -0.6690934300422668, -0.48184478282928467, -0.9147430062294006, -0.9519267678260803, 0.4517783224582672, 1.6149250268936157, -1.3088271617889404, -0.9384649395942688, 0.34573400020599365, 0.061082348227500916, -0.01838921196758747, 0.997747540473938, 0.6318857669830322, -1.9604196548461914, -0.012013306841254234, 1.7339295148849487, -0.8196774125099182, 0.6592627167701721, -0.6284396052360535, -0.037613119930028915, 0.3643108010292053, -1.2255483865737915, 0.444908082485199, -2.0336508750915527, -0.7744717597961426, 1.3148268461227417, 0.7426030039787292, -0.4871900677680969, -0.7609775066375732, -0.5615623593330383, -1.002097249031067, -0.877021312713623, -0.9658430218696594, 0.6852557063102722, 0.7079834342002869, 0.3539089858531952, -0.6546245217323303, 0.06724275648593903, -1.3710696697235107, 1.301946759223938, -0.06597889214754105, -0.45784783363342285, -0.3078877329826355, 0.6300116777420044, -0.5110270380973816, -0.4078070819377899, -1.3916642665863037, -0.15036167204380035, -0.09746373444795609, 0.290894091129303, -0.8787588477134705, 0.18494611978530884, -0.9048916697502136, 1.6847877502441406, -0.11573872715234756, -0.848558247089386, -1.7489382028579712, -0.8783060908317566, -0.6015862226486206, -0.5049757957458496, 0.778078556060791, 0.3454831838607788, -0.12823869287967682, 0.2658531665802002, 0.7595818638801575, 1.2130924463272095, 1.0829955339431763, 1.6580361127853394, -1.0037486553192139, 0.6477949619293213, 1.6798678636550903, 0.392249196767807, -0.4386315643787384, -0.9656134843826294, -1.0007668733596802, -0.5399119257926941, 1.9847110509872437, -0.6487065553665161, -1.834020733833313, 1.083489179611206, -1.695420265197754, -0.26270127296447754, 0.3680243492126465, -0.06132093444466591, 1.4110562801361084, -3.7698588371276855, -0.34522172808647156, -0.5126535296440125, 0.4570266902446747, -0.1856691986322403, -0.4237005412578583, 1.5148125886917114, 0.15549667179584503, 0.20540043711662292, -0.4147998094558716, 0.5754657983779907, -1.1521968841552734, 2.3457562923431396, -0.43609532713890076, 1.8103407621383667, 0.20010606944561005, -0.3016885221004486, -0.40359774231910706, -1.216368317604065, -1.4605748653411865, -1.8856862783432007, 0.5203262567520142, 0.04301420599222183, -1.3884800672531128, 1.683160424232483, -0.005825458560138941, 0.08404065668582916, 1.9439258575439453, -0.6534121036529541, -0.191569522023201, -1.405985951423645, -1.1043083667755127, 0.5071083903312683, -0.31811270117759705, -1.3862452507019043, -2.397686004638672, -1.356542706489563, 0.27534767985343933, -1.9307528734207153, -0.2501995265483856, -0.8025849461555481, 0.5699903964996338, 0.2416781336069107, -0.27484557032585144, -0.4592045247554779, -1.1413230895996094, 0.14331582188606262, -0.6153519153594971, 0.0348496250808239, -0.15311388671398163, -0.3521777093410492, -1.0973418951034546, 0.4536857008934021, 0.4649046063423157, -0.4056189954280853, -2.1283135414123535, -1.0774340629577637, 0.3579874038696289, 0.2773556411266327, -0.3234349489212036, -1.5331181287765503, 0.3365708291530609, -0.417594313621521, 1.6391241550445557, -1.8023085594177246, 0.23123109340667725, -1.0173935890197754, -0.8922029137611389, -2.0489184856414795, -0.4403664767742157, 0.9633758068084717, 1.549197793006897, 0.4135391116142273, 0.9067708253860474, 1.7513149976730347, -1.1299086809158325, -0.22879867255687714, -0.4149399399757385, 0.270969033241272, -1.1195907592773438, -0.08875462412834167, -0.07054505497217178, 0.07898911833763123, -1.5997531414031982, -0.22816431522369385], [0.33150342106819153, -0.1436256766319275, -1.0264778137207031, -0.424053817987442, 0.38301122188568115, 0.4040023982524872, -0.010791915468871593, -1.342113971710205, -0.6906028389930725, -0.19520562887191772, 0.19143474102020264, -2.8263449668884277, 0.8197099566459656, 0.4183046817779541, -0.6752746105194092, 1.9447753429412842, -0.9102534651756287, -1.6763397455215454, -0.7647008895874023, 0.2912401258945465, -3.013522148132324, -0.6297779679298401, -0.6212693452835083, 0.29671263694763184, -1.2580326795578003, 0.3590121567249298, -0.6011027097702026, -2.4036624431610107, -0.3087117671966553, -0.602439820766449, 1.0964657068252563, 0.8027322292327881, -0.7549687623977661, -0.7927380800247192, 0.5934979915618896, -1.0027371644973755, 0.08833613991737366, -0.58934086561203, -0.4819182753562927, 0.8762009739875793, -0.05721002444624901, 0.4866848289966583, 2.1685376167297363, 1.8862611055374146, 1.0625466108322144, 1.5827605724334717, -1.2538295984268188, 0.843356192111969, -0.8746097683906555, -0.7384538054466248, -0.7789819836616516, -1.1780768632888794, -0.32821714878082275, -1.3618316650390625, -1.8991831541061401, -0.006476443726569414, 0.8479478359222412, 2.3488998413085938, 0.19267791509628296, 0.6141761541366577, -0.5076354146003723, -0.5040138959884644, -0.3743840456008911, 0.4908084273338318, -0.6497105956077576, 1.1661368608474731, 0.38087010383605957, -0.47028523683547974, -0.2813253700733185, 0.08650226145982742, 0.9775847792625427, -0.23936772346496582, -2.5877528190612793, 1.0241495370864868, -1.0220789909362793, -0.46637651324272156, -0.5470423102378845, 0.0680152177810669, -1.8013912439346313, 1.3027575016021729, -0.15790556371212006, 0.33599796891212463, 1.4409396648406982, 0.416770875453949, 1.12430739402771, 0.5908216834068298, 1.677740454673767, 0.3936246931552887, 1.561688780784607, 1.0987989902496338, 2.324239730834961, 0.3776722252368927, -1.669586181640625, -1.029901385307312, 1.0903680324554443, -1.1413390636444092, 0.17114289104938507, 0.38303327560424805, 0.20382824540138245, 0.49929895997047424, 0.5958449244499207, 0.5051242113113403, 0.0943104475736618, -0.48801714181900024, -0.1698066145181656, 0.08279412239789963, 1.494038701057434, 0.4900830090045929, -0.8140567541122437, -0.7176334857940674, 0.7668113708496094, 0.6071223020553589, -0.2934006452560425, 1.2870101928710938, -0.4368404150009155, -0.5370557308197021, -2.3560850620269775, -0.7076297402381897, -0.6894856691360474, -0.2018435299396515, -0.2788028120994568, 1.9623264074325562, 0.48500901460647583, -1.385336995124817, -0.2030542641878128, -1.3534832000732422, -1.108934760093689, -0.13058339059352875, -0.7243941426277161, 0.20958705246448517, 0.19322563707828522, 1.8428972959518433, -0.31392061710357666, -0.7706710696220398, -0.20975784957408905, 0.0670849159359932, -0.01486058346927166, -1.3324613571166992, -1.3103961944580078, 0.17181162536144257, -0.24860574305057526, -0.12159751355648041, -1.481240153312683, 0.9502886533737183, 0.9417704939842224, -0.7632164359092712, -0.12061277031898499, -1.6992162466049194, 1.5045496225357056, -0.3519756495952606, -0.12294264882802963, 0.3306635320186615, 0.44077032804489136, 0.5294400453567505, -0.37979400157928467, 1.4236899614334106, 0.32770276069641113, 1.2875441312789917, -0.7888309955596924, 0.8487447500228882, -0.7271291017532349, -0.6306044459342957, 1.3784908056259155, 1.6304758787155151, -1.071742057800293, -0.6867581605911255, 1.431286334991455, -0.7443081736564636, -0.2503294050693512, 1.1658304929733276, -0.10613012313842773, 0.21527238190174103, -0.5536125898361206, -0.5336629748344421, 1.5140684843063354, -0.44420185685157776, 0.3955794870853424, 2.152869462966919, -0.6539287567138672, 1.0240387916564941, -0.7798001170158386, 0.3488691747188568, -0.8047471046447754, 0.25075840950012207, 1.0800282955169678, 0.4564068019390106, 0.1354581117630005, -0.33915334939956665, -1.6684222221374512, -1.7219350337982178, 0.8497315049171448, 0.29740267992019653, -0.4282459318637848, 0.11970869451761246, -0.7503269910812378, 0.64248126745224, -0.9622626304626465, -1.1051901578903198, -0.8348581194877625, -0.553435206413269, -0.6114373207092285, 0.5487080216407776, -0.1825096160173416, -1.5902225971221924, -0.6531920433044434, -0.4274541437625885, -0.8987407088279724, 0.739465057849884, 0.8285098075866699, 0.5204528570175171, -0.5696636438369751, -0.5496056079864502, 0.48058363795280457, 0.7354884743690491, -0.13006478548049927, 0.720022439956665, -0.00026660296134650707, -1.615618348121643, 1.470402479171753, -2.192819356918335, -0.23109149932861328, 1.529072880744934, -1.5936082601547241, 1.8452702760696411, -0.6055406928062439, 0.790377676486969, 0.19502943754196167, 0.3683033883571625, -0.06099487468600273, -0.45265963673591614, 1.592168927192688, 0.8549442291259766, 1.4626336097717285, -0.5199607014656067, 0.46379610896110535, 0.45563751459121704, -0.016481604427099228, 1.048516035079956, -0.03934479504823685, 0.6189416646957397, 0.0006415867828764021, -0.38768911361694336, 0.2795601487159729, 0.04777213931083679, 0.41930630803108215, 0.43551966547966003, -0.7253901362419128, 0.3318006992340088, -1.0714890956878662, -2.383319139480591, -0.5787470936775208, -0.12339510023593903, 1.5956734418869019, -1.732068419456482, -0.4423305094242096, 1.4869747161865234, 1.0784835815429688, -0.575407087802887, -0.2963694632053375, 1.7834700345993042, 0.2080542892217636, 0.6712168455123901, -0.24189430475234985, -0.06995775550603867, -0.10867013037204742, 1.1817080974578857, 1.5550404787063599, -0.8845825791358948, 0.8294339776039124, -2.1144800186157227, -0.40349438786506653, -0.25180721282958984, 0.8932251930236816, 1.7916100025177002, 0.5388234853744507, -1.240332007408142, -0.48649853467941284, 0.8810459971427917, 0.7464959621429443, 0.8168764710426331, 0.31455090641975403, 1.0169216394424438, 0.6127755045890808, -1.783062219619751, 1.2121130228042603, 3.107595920562744, -0.4137570858001709, -0.8503462672233582, 0.4879222810268402, -0.10629014670848846, -0.5557985305786133, 0.9669363498687744, -1.1081111431121826, 0.27437958121299744, 0.409560889005661, 1.2592812776565552, 0.4802270829677582, -1.2322282791137695, -0.19147086143493652, 1.476881980895996], [-1.7241863012313843, 0.40506988763809204, -0.976435661315918, -1.798552393913269, 0.48810693621635437, 1.4130553007125854, 1.680555820465088, -0.659461498260498, -0.9366289377212524, 0.5856395959854126, -0.6991536617279053, 1.448742151260376, -0.257243812084198, 0.05378939211368561, -0.7705715894699097, -0.01945963129401207, 2.2748563289642334, 0.8136905431747437, 1.3505867719650269, -1.5597177743911743, 1.0160109996795654, -0.027595803141593933, -0.7946693897247314, 0.47528454661369324, -2.2039968967437744, -0.07346569746732712, -0.5236804485321045, -0.5785114765167236, -0.27577707171440125, 0.7182080745697021, -0.7265410423278809, -2.2826764583587646, 0.6586059331893921, 1.7909883260726929, -1.8714882135391235, 0.8959625959396362, 0.6783931851387024, 0.9475836157798767, -0.3662050664424896, 2.200839042663574, 1.7079888582229614, 0.9315029382705688, 0.6277754306793213, -0.0751071646809578, -0.4444066286087036, -0.7449652552604675, 1.3227880001068115, -0.5813480615615845, -0.1910007894039154, 0.8020133376121521, -0.756081759929657, -0.3644997179508209, 0.631036639213562, -0.505018949508667, -0.09769579768180847, -1.045555591583252, 0.9689565896987915, -1.0081576108932495, -1.4599405527114868, -1.9555177688598633, -0.505788266658783, 0.03751811012625694, 0.2148514986038208, -0.002180845243856311, -0.4719354808330536, -0.015283431857824326, -1.2143335342407227, 0.27213823795318604, -0.517471194267273, 1.6975688934326172, -0.18966062366962433, 0.3089419901371002, 1.4270539283752441, -0.0094487639144063, -0.60909104347229, -0.1366899460554123, 0.5964035987854004, -0.7645430564880371, 0.300504207611084, 1.7521672248840332, -0.12339621782302856, -0.6161908507347107, 0.26886194944381714, 2.093194007873535, -0.38005825877189636, -1.4320791959762573, -0.8714199662208557, 1.575085997581482, 0.18471746146678925, 1.7579927444458008, 1.431528091430664, -1.992812156677246, -1.4215630292892456, -0.3771854341030121, 0.41944241523742676, 1.0965855121612549, -0.46699729561805725, 0.2825588583946228, -0.3611282408237457, -1.959875226020813, 0.46844682097435, -0.8312533497810364, 0.22330302000045776, 0.9563981294631958, 1.3522788286209106, 1.4321117401123047, 1.0821651220321655, 0.1922137588262558, -0.2868097722530365, 0.34943583607673645, -0.6337313652038574, 0.7308960556983948, -2.058990478515625, 0.21311576664447784, 1.6672585010528564, 2.0513975620269775, 1.8058382272720337, -0.43103650212287903, -0.9380099773406982, 0.6410810947418213, -0.35647493600845337, -3.0759031772613525, -0.9465068578720093, -0.733065664768219, 0.012620139867067337, -0.9046957492828369, 0.32497432827949524, 0.8311273455619812, 0.2259250432252884, -0.12259630858898163, -0.7719270586967468, -0.7844324707984924, 1.4112591743469238, 0.9018160700798035, -0.28125351667404175, -0.6876636147499084, 0.03867170587182045, 0.7806745171546936, 0.8499285578727722, -0.2535701394081116, 0.40102195739746094, 1.0656529664993286, 0.4237167537212372, -1.1286699771881104, 0.9529356360435486, 1.1298619508743286, -0.9960474371910095, -0.6863657236099243, 2.2166740894317627, 0.42509689927101135, 0.5313854813575745, -0.46610739827156067, 0.34066998958587646, -0.3179366886615753, -1.9222114086151123, -0.4033958315849304, -1.1273168325424194, -1.5194631814956665, -0.9469358325004578, -0.7156862616539001, 1.438980221748352, 0.0911349505186081, -1.4279693365097046, 1.1058136224746704, -0.06082450598478317, -1.331329584121704, -0.13873250782489777, 0.5463246703147888, -0.3670586943626404, -1.862282633781433, -0.8807246685028076, -0.744673490524292, 0.1697358638048172, -0.9913303256034851, -1.0110219717025757, -1.6255954504013062, 1.48611319065094, 2.7814810276031494, 0.18458323180675507, -0.14495128393173218, 0.26529574394226074, -0.3488384783267975, -0.3701537251472473, -3.101332664489746, -1.3064934015274048, 1.3562217950820923, -0.4030921459197998, 0.584450900554657, 0.21816366910934448, -0.09939071536064148, 0.3507213294506073, -0.6206477880477905, 0.7329342365264893, 0.7888113856315613, 0.563909113407135, -1.3509999513626099, 0.6428316831588745, -1.0731662511825562, -0.4484259784221649, -0.7798190712928772, -1.5153751373291016, -0.037522125989198685, 0.8771500587463379, -0.7972128987312317, 0.6024941802024841, -1.0997151136398315, -1.2003732919692993, 0.016742292791604996, -1.3129570484161377, -0.25070950388908386, 0.530328094959259, -1.2588545083999634, -0.008147314190864563, 2.1381819248199463, -0.8734464645385742, -1.1441755294799805, -0.2115897536277771, -1.3211607933044434, -0.13439863920211792, -0.5255006551742554, 0.19670690596103668, -2.080930709838867, 0.3186061382293701, 0.5982359051704407, -1.4149188995361328, 1.1277000904083252, -0.44827449321746826, -0.0022386875934898853, 2.2507152557373047, 0.5820866823196411, 0.8875210881233215, 0.3491341471672058, -0.0019770748913288116, 1.1999943256378174, 0.3086327016353607, 1.4192098379135132, -0.3579765856266022, 0.3042067885398865, -1.1382865905761719, -0.12971654534339905, 0.6912039518356323, 0.06224338710308075, 0.5703085660934448, -1.2438130378723145, 2.1838746070861816, 0.17216438055038452, -1.9255461692810059, -0.19632156193256378, 1.4856253862380981, -0.09360180795192719, 0.44936037063598633, -1.3643097877502441, -0.9244535565376282, -0.47950276732444763, 0.5411646962165833, -1.8553717136383057, 0.08951134979724884, -0.1489989310503006, 1.0011180639266968, -0.581603467464447, -0.320455402135849, 2.0202839374542236, 0.8058193922042847, -1.1020501852035522, 0.6114619374275208, 1.476980447769165, 0.5613487958908081, -0.6218686699867249, -0.09409181773662567, -0.9461848139762878, 0.023776037618517876, 0.6431962847709656, -0.06681030988693237, 1.091988444328308, 2.3301360607147217, 0.7071164846420288, -0.4673960208892822, 0.016046518459916115, -0.3602459132671356, -0.4627176523208618, 0.22475303709506989, 0.26538950204849243, -1.2359529733657837, -0.07351992279291153, -0.634139358997345, -0.9036584496498108, -0.9228653311729431, 0.6441341638565063, -0.06870951503515244, 1.549090027809143, 0.7427892684936523, -1.2959052324295044, 0.3985123336315155, 0.9512851238250732, -1.41395902633667, -2.084317207336426, 0.2281130850315094, 0.4824235439300537, -0.1427757889032364, -1.331636667251587], [-0.05435070022940636, 0.2926222085952759, -1.5907663106918335, -0.9259706139564514, -1.51779305934906, 2.6440422534942627, -0.6526506543159485, 0.9792293310165405, -0.00240257172845304, 0.8062311410903931, -1.6789606809616089, 1.3563491106033325, 0.6431280970573425, 0.6019789576530457, -0.6278546452522278, 0.9902951121330261, -0.41666582226753235, -0.07240164279937744, -1.6182657480239868, -0.12247385829687119, -0.369305819272995, 0.8453190922737122, -0.810249924659729, -1.7556664943695068, -0.2962118983268738, 0.5463874340057373, -1.6826506853103638, 1.5362739562988281, 0.8889375329017639, -0.14078950881958008, -0.5012505650520325, -2.069718360900879, 1.401473879814148, -1.3364171981811523, 0.6566940546035767, 0.534171462059021, -1.5552088022232056, 0.6251029968261719, -0.7501797676086426, 0.8120778203010559, 1.422530174255371, -1.3605296611785889, -1.56815767288208, -0.6600813865661621, 2.8740956783294678, -0.19746588170528412, -0.4204431474208832, 0.3087126612663269, -0.26115429401397705, 0.7066176533699036, -0.12656421959400177, -0.48329606652259827, 0.7063285708427429, 0.3717152774333954, 1.0939072370529175, 0.4643104374408722, 1.096258521080017, -1.2096788883209229, 2.5914535522460938, 0.7412482500076294, 0.4409237802028656, 0.08545525372028351, -1.1408467292785645, -1.2675511837005615, -1.2471375465393066, -0.8229983448982239, 0.8570814728736877, 1.0476691722869873, -0.9337556958198547, -0.02461356297135353, -0.9991707801818848, -0.30128511786460876, -0.10303808003664017, -1.4484986066818237, -0.17266952991485596, 1.377615213394165, 0.02104141190648079, -1.3373098373413086, -0.10209476947784424, 0.0536687895655632, -0.1554970145225525, -1.717729091644287, 0.2978200614452362, -0.030176112428307533, -0.08903390914201736, 0.9042502641677856, -0.6391879916191101, -0.9134840369224548, 1.450596809387207, 0.17493845522403717, -1.0579770803451538, 0.9949460625648499, 0.37530943751335144, 1.3564770221710205, -0.9661661982536316, -0.6079967021942139, -0.4034421145915985, -0.8733137249946594, -1.236222743988037, 0.9553175568580627, 1.6890842914581299, 3.0915582180023193, -0.6396300196647644, -0.3311251103878021, 0.14242909848690033, -0.6160916686058044, 1.3052432537078857, 0.011689026840031147, 0.7620282173156738, 0.02510504610836506, 0.7581964731216431, -0.11680872738361359, 0.9238771796226501, 0.3617713451385498, -0.7718366384506226, 0.4926166832447052, -1.9702098369598389, 0.8444082140922546, 0.18174800276756287, -0.23604483902454376, -1.1165298223495483, 2.5481560230255127, 0.8448036313056946, -0.5743284821510315, 0.08484680205583572, -0.6367778182029724, -0.15808752179145813, -1.4649163484573364, 0.2971944808959961, 0.16779936850070953, -0.31502631306648254, 0.36968111991882324, 0.9453946948051453, 0.8973290920257568, 0.1398644745349884, -0.1357918232679367, -0.4318934381008148, -0.9386693835258484, 1.5786367654800415, -0.9167689681053162, 0.48180222511291504, 1.3320292234420776, -0.9051059484481812, 1.4085406064987183, -1.372232437133789, 1.5908704996109009, -0.548999547958374, 0.46723437309265137, -0.6118355989456177, 2.113171339035034, -0.367153525352478, 0.16051535308361053, 0.7023967504501343, 0.08754075318574905, -0.06169315055012703, -0.4271862208843231, -0.729515552520752, -0.5166023969650269, 1.2068623304367065, 1.3346558809280396, 1.2757772207260132, -0.032873980700969696, 0.6542539000511169, -1.090847134590149, 0.19666828215122223, 1.613647222518921, 1.3652091026306152, 0.014529840089380741, 2.2672879695892334, -0.7266219258308411, 0.42637157440185547, 0.3131982386112213, -1.4515949487686157, 0.3735424876213074, -0.8601981401443481, -0.7048243284225464, 1.2265070676803589, -0.1528012901544571, -1.2742372751235962, 1.044458270072937, 1.1760096549987793, 1.5712047815322876, 0.828948974609375, -0.6466661095619202, 1.476901888847351, -0.0516025573015213, 1.8003649711608887, 1.5004582405090332, 1.6078816652297974, 0.46856939792633057, -1.0282756090164185, 0.4520334303379059, 2.459412097930908, 0.40808600187301636, -0.9877704977989197, -0.9863102436065674, 0.18456421792507172, 1.2100468873977661, 0.17650462687015533, 0.9797136187553406, 0.45933082699775696, 0.08130469173192978, 0.05506489425897598, 1.5277994871139526, 1.346967339515686, 2.4392547607421875, -0.705596923828125, -1.1711196899414062, -0.7432036399841309, -0.3099297881126404, -1.0761605501174927, -1.3953012228012085, -0.8908945322036743, 0.615427553653717, -0.15971516072750092, 0.7548506259918213, 1.5031561851501465, -0.5370931029319763, 0.8234989047050476, -0.7344537377357483, 0.919999361038208, -0.45763611793518066, 0.08629096299409866, -0.9344130158424377, 0.7418006658554077, 0.19845741987228394, 0.2896767556667328, -1.5173139572143555, 0.17484150826931, -0.12223473936319351, 0.9154847264289856, -0.9864183068275452, -0.04517050087451935, 0.13263025879859924, 1.0735113620758057, -0.004739133175462484, 1.173017144203186, 0.7747336626052856, -0.23504403233528137, -0.8246788382530212, 1.0045270919799805, -0.1885245144367218, 0.5469042658805847, 0.4796040952205658, 1.3915431499481201, 0.9284085631370544, -0.3523046374320984, -0.15929119288921356, -0.13629159331321716, -1.5532878637313843, -1.3071411848068237, 0.344463050365448, -1.7056553363800049, -0.1763402670621872, 0.15084798634052277, -1.0379549264907837, -0.6654723882675171, -0.4417177140712738, 2.309311628341675, 1.9007331132888794, -0.9138987064361572, -0.20978288352489471, -1.337746024131775, -1.335172414779663, -0.7086589336395264, -0.23033998906612396, 1.851824402809143, 0.6919708251953125, -0.3092125356197357, -0.6327904462814331, -0.9792717099189758, 0.869974672794342, 1.5630093812942505, -0.970883846282959, -0.20888443291187286, 1.489758849143982, 0.6973438858985901, 1.7355748414993286, 2.2349226474761963, 1.0458613634109497, 0.9710589647293091, 0.08405663073062897, -0.6578648090362549, -0.031194154173135757, 0.1527419090270996, -0.23596367239952087, -0.7261683344841003, 0.5064290761947632, -0.45935091376304626, 0.7142818570137024, -0.4442920386791229, -0.7300320863723755, 2.1917648315429688, 0.3042981028556824, -1.285136103630066, -0.8841874599456787, -1.6316853761672974, 0.08548921346664429, 0.1870938241481781, -1.2885737419128418], [-0.34215089678764343, 2.24365496635437, 0.9964489340782166, -0.47457921504974365, -0.5522544980049133, -0.13964475691318512, 1.7740652561187744, -0.07695086300373077, -1.9403362274169922, 0.9404066205024719, 0.20429779589176178, -1.482364296913147, -0.7194108963012695, -1.0544031858444214, -2.901576280593872, -0.4953877329826355, 0.005170921329408884, -0.7166117429733276, 0.21433693170547485, -0.6526828408241272, -2.164233922958374, 0.8721636533737183, -1.6054314374923706, -0.054642319679260254, -0.1608952134847641, 0.6449838876724243, -1.5661119222640991, -1.1282906532287598, 1.571401596069336, 0.7296053767204285, -0.659209668636322, -0.04413306340575218, -1.2556403875350952, 2.0790274143218994, 0.8765350580215454, -1.242861032485962, 0.2778691053390503, -0.30609503388404846, 0.2205635905265808, -0.7085490822792053, -0.3847084641456604, 0.5724511742591858, -0.13404330611228943, -1.0098357200622559, -0.3469356894493103, 0.9960551261901855, 0.205821231007576, 0.10887129604816437, 0.6618379354476929, -0.38464927673339844, -1.2174873352050781, -1.1208268404006958, 0.2597580552101135, -0.16100725531578064, -0.35982048511505127, 0.6456031799316406, 1.7631466388702393, -0.9417226910591125, -0.4346548914909363, 0.024094292894005775, -0.6879541277885437, -0.24096739292144775, 0.2953028082847595, -1.5058248043060303, 0.3583216071128845, -1.5465646982192993, -0.5897729396820068, -0.6325798630714417, -0.4391680657863617, -1.1694612503051758, -0.26224562525749207, -0.610633909702301, -1.5801678895950317, -1.919324278831482, -0.0561806783080101, -0.34808385372161865, 1.3160063028335571, -0.940101146697998, -0.8250529170036316, 0.6617715358734131, -0.8825420141220093, 0.16654464602470398, 2.1968302726745605, 2.0076711177825928, 0.5415550470352173, 0.07499805837869644, 0.34865352511405945, -0.7479061484336853, -0.464242547750473, 2.05739688873291, 0.8929060101509094, -0.2586211562156677, -0.3305874168872833, 0.7370558977127075, -0.0052634975872933865, -0.8180185556411743, -0.3342891335487366, -1.5476118326187134, -1.046409249305725, 1.134310245513916, -0.42756491899490356, 0.792430579662323, -2.3504881858825684, 0.4305979311466217, -0.00614865543320775, -0.7746885418891907, 0.7256263494491577, -0.8485739827156067, -0.4523274898529053, 0.8241286873817444, -0.7478727698326111, -0.6144819259643555, 0.826088547706604, 0.06208423152565956, 0.07912951707839966, -0.6632738709449768, 0.6295651793479919, -1.7514145374298096, 0.009497368708252907, 0.8764175772666931, -0.7782032489776611, -1.0348976850509644, -0.03925590589642525, 0.40396350622177124, 0.0004278174019418657, 0.6467652916908264, -0.2536061108112335, -1.7541096210479736, -0.9796198010444641, 0.07016482949256897, -0.18521353602409363, 0.678625762462616, -0.16929665207862854, 1.5087847709655762, -0.7873971462249756, -1.5681928396224976, 0.1334581971168518, -0.4438188970088959, -0.6340293884277344, -1.157561182975769, -1.1170039176940918, 0.4898660182952881, -0.8231726288795471, -0.899265706539154, 1.3318853378295898, -1.569267988204956, -1.2641289234161377, 0.7113485336303711, -0.1028653010725975, -0.46085453033447266, 0.5714000463485718, -0.10255962610244751, 0.6365714073181152, 1.2514145374298096, -0.27338463068008423, 1.4018958806991577, 0.9919623732566833, -1.221478819847107, -0.9334617853164673, -1.050818920135498, -0.43850645422935486, -0.3256792724132538, -0.3387095034122467, 0.863350510597229, -2.4097747802734375, -0.7968237400054932, -0.06103306636214256, -0.6108413934707642, -0.7594739198684692, 0.1223510280251503, -0.30866366624832153, -0.22475898265838623, -0.6748291254043579, -2.0196022987365723, -0.4194430708885193, -0.41162389516830444, 0.948766827583313, 1.0560524463653564, 0.7658759355545044, 0.3670208156108856, -0.32197242975234985, 0.6812542080879211, -1.2387243509292603, 1.4901669025421143, 0.019337031990289688, 1.4427103996276855, 0.6762975454330444, 0.8675336837768555, 1.187791109085083, 0.04165585711598396, 0.0408211387693882, -1.3314982652664185, -0.6222134232521057, 0.519690990447998, -1.0839601755142212, 1.288973331451416, 1.0556519031524658, -0.7128597497940063, -2.202075481414795, -1.136581301689148, 0.6513632535934448, 0.4386517107486725, 0.002124152844771743, -0.59742671251297, -1.7937049865722656, 0.5708079934120178, -1.6273597478866577, 0.29395049810409546, -0.9987911581993103, -0.41030746698379517, -0.42622968554496765, -0.2050330489873886, 0.01729532890021801, -0.24985583126544952, 2.0039138793945312, -0.05094938352704048, 0.5250668525695801, 0.681928277015686, 1.8006190061569214, 0.0011236784048378468, -0.8756546378135681, -0.774507462978363, -1.826529622077942, 0.5923091173171997, -0.892939567565918, 2.0385000705718994, -1.0240942239761353, -0.38218721747398376, 1.2624385356903076, -2.0642688274383545, -0.5382205247879028, 0.1955065131187439, -1.7973859310150146, -0.053042687475681305, 0.41103941202163696, 0.511325478553772, -1.9100764989852905, 0.34957775473594666, 1.3018033504486084, -0.3669634759426117, 0.12408732622861862, -0.4906078279018402, -0.8831238150596619, 0.701777458190918, 0.3824490010738373, 0.7910467386245728, -0.6440145373344421, -1.440176010131836, 0.5425933599472046, -1.3069299459457397, 0.23049359023571014, -0.08463380485773087, 0.3309297561645508, -1.3999079465866089, -0.6858474016189575, -1.350478172302246, 0.4305570721626282, -0.11171628534793854, 0.003545572282746434, 0.5963732004165649, 0.4962816834449768, -0.08825330436229706, -1.1019773483276367, 0.17420941591262817, -0.5741553902626038, 1.0867210626602173, -0.4405352473258972, -0.46481987833976746, -0.0011113944929093122, -0.4670235812664032, -0.19409888982772827, -0.6920803785324097, 0.4689730703830719, 0.6838175654411316, 0.32191938161849976, 0.4540383815765381, -1.6348382234573364, 0.8050218820571899, -0.6025245785713196, -1.1234301328659058, 0.7497122287750244, 0.3101853132247925, 0.5399850606918335, -0.8611730933189392, -0.4875410497188568, 0.5059506297111511, 0.3428189754486084, -0.029182514175772667, 0.2601897120475769, -1.0822027921676636, 1.051211953163147, 0.7136639952659607, -0.6177632212638855, 0.9872494339942932, 0.5112733244895935, -0.9966832399368286, 1.8557677268981934, 0.22379182279109955, -0.3003593981266022, 0.3745344579219818], [-2.459071159362793, -0.9229342341423035, -0.2689850628376007, 0.1485465168952942, 2.2079250812530518, 0.3806248903274536, 0.18016009032726288, 1.387402057647705, -1.948903203010559, -0.7164127826690674, -0.10240121930837631, 0.668340802192688, 0.21508976817131042, -1.0891151428222656, 0.3135031461715698, 0.4538831114768982, -0.39717912673950195, -0.006415612529963255, -0.12921282649040222, 0.9024689793586731, 1.2490214109420776, -0.14114050567150116, -0.8390178084373474, 0.12920987606048584, 0.9954082369804382, -0.9753165245056152, 0.058959271758794785, -1.1417055130004883, -0.325927197933197, -0.773597002029419, 0.7301180958747864, -1.1807594299316406, -1.2376347780227661, 0.3970932364463806, -1.874927282333374, -1.113779902458191, -0.557177722454071, 1.151517629623413, -0.12246628105640411, -0.2962014675140381, -2.3141486644744873, 0.6572689414024353, -0.2738392651081085, -1.6718887090682983, -0.6340168714523315, -0.757810652256012, -0.6454786658287048, 0.11806929111480713, -0.41150012612342834, -0.6216855645179749, 0.4267413020133972, -0.4259764552116394, 0.8197950720787048, 0.4300505220890045, 1.061785340309143, -1.5938410758972168, -0.8035948872566223, -0.2321677953004837, -0.13588514924049377, -0.01721923239529133, 0.7348760962486267, -0.9178009629249573, 0.09107948839664459, -0.9524796009063721, 0.8620911240577698, 1.3889423608779907, 0.2578139305114746, 0.27118271589279175, -0.5824150443077087, 0.7154132127761841, -1.323143482208252, -2.471909523010254, 1.574876070022583, -0.43051445484161377, 0.6866831183433533, -1.2362072467803955, -0.26264938712120056, 0.6405274271965027, -0.43972116708755493, -0.5137327909469604, -0.4933745563030243, 0.8849276900291443, 1.5903565883636475, 0.831132709980011, 1.0333666801452637, -1.0520391464233398, -2.2074806690216064, -1.3897382020950317, 0.2531796395778656, -0.2115081548690796, 0.7177865505218506, -0.1837511956691742, -0.7346948981285095, -0.8533173203468323, 0.5555047392845154, 1.3148818016052246, -1.8569128513336182, -0.5017080307006836, -0.17250558733940125, -0.40999332070350647, 1.5315821170806885, -1.034012794494629, -1.1040278673171997, -0.1059204712510109, -0.4232330024242401, -0.08431926369667053, -1.0901802778244019, -0.3234078288078308, -0.09719850867986679, 2.7371714115142822, -0.12216545641422272, 0.01863233558833599, 1.3601329326629639, 0.1421123445034027, -0.5831446647644043, 0.2740655839443207, -0.7877214550971985, -0.5792295336723328, -0.23099879920482635, 0.954997181892395, -1.0138311386108398, -1.5421180725097656, -1.4384198188781738, -0.48378732800483704, 0.1766771376132965, 0.860569953918457, 1.043478012084961, 0.3595735728740692, -1.9170161485671997, -1.2976744174957275, 0.5677832365036011, 0.25004851818084717, 0.3581504821777344, 0.5678176283836365, -0.1636444479227066, 1.6761480569839478, -0.5575597882270813, -0.4996170103549957, 1.0890426635742188, -1.2741819620132446, -0.8325955867767334, 0.46108296513557434, 0.5592873096466064, 0.5189595818519592, -0.07628293335437775, 0.020648900419473648, 0.5564427375793457, 0.4004688858985901, -0.4731287360191345, -1.6411638259887695, -0.34930112957954407, 0.7731025815010071, 0.8432562947273254, 1.2244716882705688, -0.45639801025390625, 0.6357289552688599, -0.7591514587402344, -0.3441580832004547, 0.246872216463089, -0.1905377060174942, -1.0593814849853516, -0.17407266795635223, 0.25705546140670776, -1.6611411571502686, -1.9165526628494263, -0.1833648532629013, 1.3030787706375122, 0.35256528854370117, 0.9182358384132385, -0.34176018834114075, 1.1947789192199707, 1.4199275970458984, -3.2971489429473877, -0.9292426705360413, 0.4315304458141327, 0.1755870133638382, 0.8045783042907715, -0.7058883309364319, 2.5251173973083496, 0.7205737829208374, -0.348043292760849, -0.6987732648849487, -0.5802100300788879, -0.2745639979839325, 0.8962138295173645, 0.8623387813568115, 2.163429021835327, 0.6002500057220459, -0.11722537875175476, -0.6345817446708679, 2.0044572353363037, -1.2279446125030518, 0.7002118825912476, 1.4081199169158936, 0.22321352362632751, -1.5214980840682983, 1.34575617313385, 2.5403311252593994, -1.344752550125122, -0.276596337556839, -0.04571797326207161, -0.6593062281608582, -1.4065805673599243, 0.7341833114624023, 0.09783030301332474, -1.2670812606811523, 1.101585865020752, -1.1631696224212646, -0.13029786944389343, -1.2910717725753784, -1.0778387784957886, 0.43465694785118103, 1.1633765697479248, 0.4858190417289734, 0.42085006833076477, 0.48236891627311707, -0.47826528549194336, -1.2438479661941528, 0.16090302169322968, 0.9610068798065186, 0.024536501616239548, -0.10706783831119537, 0.4063740372657776, -0.3122347593307495, -0.41556796431541443, 0.06531544029712677, -1.3474116325378418, 0.8910629153251648, -0.7193474173545837, 1.0399065017700195, 1.2687783241271973, -1.218144416809082, 0.0038051363080739975, 1.2386577129364014, -0.03301437199115753, -0.9408756494522095, -2.0406813621520996, 0.8932874202728271, -1.1072686910629272, -0.44765785336494446, -0.5848106145858765, 0.14421901106834412, -1.6299830675125122, -0.539018452167511, 0.5533908605575562, 0.26739317178726196, -0.8262589573860168, -0.5064978003501892, 0.08564946800470352, -1.1397706270217896, 0.015990447252988815, 0.6790599226951599, 0.2727571725845337, 2.3637466430664062, -0.289109468460083, -1.0189119577407837, -1.1839545965194702, -0.5957890748977661, 0.6654438972473145, 0.6204695105552673, -0.2443303018808365, -0.4356369972229004, -1.3020113706588745, 1.8152117729187012, 1.3447887897491455, 1.1060799360275269, -0.1367826610803604, -0.7804931998252869, 0.5800886154174805, 0.09543762356042862, -0.42600399255752563, 0.06383851915597916, -1.208229422569275, -1.3260092735290527, -0.4135482907295227, -0.8263230919837952, 0.3777998387813568, 1.677762508392334, 0.6365565657615662, 0.6408652067184448, 0.7666777968406677, 0.1486113965511322, 0.9067635536193848, 1.4026072025299072, 1.0577086210250854, 1.2481614351272583, 0.6544604301452637, -0.7787629961967468, -1.0746753215789795, -0.6074216365814209, -0.6036654114723206, 1.478386402130127, 0.5676001906394958, -1.376442551612854, 0.9239557981491089, 0.9703013896942139, 0.0471230186522007, -0.600095808506012, -0.4245838522911072, -0.6340200304985046], [0.5705490112304688, 1.3388458490371704, -0.3931056261062622, 0.7371383905410767, 0.8388689756393433, 2.46469783782959, 0.4970496892929077, 0.03102339617908001, -0.6907879114151001, -0.7471487522125244, -0.06584353744983673, -0.8608716130256653, 1.173294186592102, 0.617209255695343, -0.7936033606529236, 0.9834043383598328, 0.4252046048641205, 0.17294861376285553, -0.011084345169365406, 0.8240121603012085, -0.7154594659805298, 1.1051608324050903, -0.5743870139122009, -0.40657544136047363, 0.7394018173217773, 0.4785609543323517, 0.569595456123352, 0.7895422577857971, -1.9665695428848267, 0.9188774824142456, 0.5923965573310852, 2.1213083267211914, -0.5845455527305603, -0.497775137424469, -0.6503100395202637, -0.5574747920036316, 0.6474699378013611, 0.7441611886024475, 0.6692935228347778, -1.3080424070358276, 1.3499877452850342, -0.2206866592168808, -1.8843263387680054, -0.16327734291553497, -0.9401880502700806, 1.4226150512695312, 0.31816187500953674, -0.9253177046775818, -2.301297664642334, 0.6125373244285583, -1.8667508363723755, 1.07488214969635, -0.6387810111045837, 0.6689046621322632, 1.1512255668640137, -0.671780526638031, 0.049474433064460754, 1.3249365091323853, -0.49851471185684204, 1.0368783473968506, 0.2116488367319107, 0.13613957166671753, 0.16773393750190735, 1.7643349170684814, -0.5371048450469971, 1.929911732673645, -0.6708797216415405, 0.38540762662887573, 0.29869818687438965, 1.7156187295913696, 0.46439477801322937, 1.514312744140625, 2.4114086627960205, -0.6314411163330078, -2.6450421810150146, 1.5915217399597168, -1.0319207906723022, -0.9162907004356384, 0.5590342879295349, -0.35242947936058044, 0.27702298760414124, -0.7238738536834717, -1.2682501077651978, 0.603581964969635, 0.1572667956352234, 0.8775978684425354, 2.2756340503692627, 0.033335331827402115, -1.1043832302093506, 2.5558362007141113, 0.9913834929466248, -0.8157401084899902, 1.3828476667404175, 0.019237730652093887, -0.43899673223495483, 1.1951900720596313, 1.4426966905593872, 0.35406842827796936, 0.561231791973114, 0.7266876697540283, 0.4617128372192383, -0.5009359121322632, 0.6038616299629211, -0.27634942531585693, -0.08171036839485168, 0.6274507641792297, -1.7021151781082153, -1.2782549858093262, -1.022605299949646, -0.5342766642570496, -1.477713942527771, -1.7857822179794312, -1.6937172412872314, 1.2246769666671753, -2.3311922550201416, -0.4547063112258911, 1.4782557487487793, -0.3157506585121155, 0.4792867600917816, 0.6826397776603699, -1.5396984815597534, 0.5337409377098083, -1.6745567321777344, -0.3289506733417511, 1.2511299848556519, 0.7616750597953796, -0.2329082041978836, 0.42158401012420654, -1.3190250396728516, -0.30240756273269653, 0.25800248980522156, 0.4445284903049469, -1.1081836223602295, 1.749732255935669, 1.9482247829437256, 1.2579182386398315, 0.6773473620414734, -0.916203498840332, 0.46837326884269714, 0.017200445756316185, -0.3502509891986847, -1.685159683227539, -1.3257092237472534, 0.3019007444381714, 0.012706168927252293, -0.10315165668725967, 1.9088950157165527, 0.23127251863479614, -0.36921700835227966, -0.012203269638121128, -0.4937564730644226, 0.7200827598571777, -0.3827073872089386, -1.0348758697509766, -0.12836554646492004, -0.8854147791862488, 0.39117899537086487, 1.2159748077392578, 0.9054643511772156, 1.210221290588379, 0.00474830437451601, -0.6418556571006775, 0.4785446524620056, 0.1175612360239029, -0.5382339954376221, -0.9097223281860352, -0.1687469333410263, -0.8854672312736511, 0.3036319315433502, -0.8314447402954102, 0.5657238960266113, -1.007175326347351, -1.0315141677856445, 0.3909602761268616, -0.4957268536090851, -1.0708974599838257, 1.583744764328003, 1.277340054512024, 0.4002343416213989, 0.8621112704277039, 0.08186155557632446, -0.7395370602607727, -0.7369070649147034, 0.568260133266449, -0.9026170969009399, 0.10352011770009995, -1.106027603149414, -0.5140567421913147, 0.8766440749168396, -2.3078463077545166, 0.3229368031024933, 0.9810616970062256, -0.046284496784210205, 0.3478424549102783, 0.316291868686676, 1.420859932899475, 0.35474392771720886, 0.8336381912231445, -0.69866943359375, 0.11261507123708725, -1.416115641593933, 0.1990966647863388, -0.2589915096759796, -0.7042200565338135, -1.245572566986084, 1.0435158014297485, 2.442756414413452, 0.33365780115127563, -1.043697714805603, -0.03473236411809921, -1.568505883216858, -0.2314995974302292, 0.027073604986071587, -0.8775852918624878, 0.26235905289649963, 0.24541626870632172, -0.33624741435050964, -1.8000441789627075, 0.6105630397796631, 0.3797488808631897, 0.05369660630822182, 1.6572893857955933, 1.1489144563674927, 0.5315638780593872, -0.48113152384757996, 1.0983681678771973, 0.7397705316543579, 0.14187578856945038, 0.10577352344989777, 0.2862229347229004, 0.19169405102729797, 1.984589695930481, 0.32269617915153503, -0.7622067332267761, 0.9161432385444641, -0.44571617245674133, 0.07163453847169876, 0.6499059796333313, -0.19346585869789124, -1.5042725801467896, -0.3596280813217163, 0.6871535181999207, -1.176694393157959, 0.1749480962753296, -0.5024895668029785, 0.6731848120689392, -1.785137414932251, 1.015541672706604, -0.24463079869747162, -0.6747689843177795, 0.4183279275894165, -0.023105856031179428, -0.67830890417099, -1.1707706451416016, 0.43310096859931946, 0.6493839621543884, -0.1494944840669632, 0.047683507204055786, 0.34802982211112976, -0.952336847782135, -0.5132095217704773, 1.183998465538025, -1.278517484664917, -0.5782740116119385, 1.6826777458190918, 0.38606396317481995, -1.167782187461853, 0.5023852586746216, 1.8790996074676514, 1.7675790786743164, 3.292524576187134, -0.25936752557754517, 0.06526102870702744, 0.33777767419815063, 1.728846549987793, -1.0842409133911133, 0.905993640422821, -0.36725279688835144, 1.814658761024475, -1.5167748928070068, -1.6435147523880005, -0.9080039262771606, 0.24635328352451324, -1.9898821115493774, 1.0773298740386963, -0.47149738669395447, 1.8350452184677124, -0.27303239703178406, 1.8598145246505737, 1.526832103729248, 0.8271008729934692, 0.8515297770500183, 0.462584912776947, -0.42529547214508057, -0.9252541661262512, 0.3012339770793915, 0.01922980695962906, -0.9160455465316772, 1.1659932136535645, -1.0805846452713013], [-0.7305130958557129, -0.5025681257247925, -0.22422631084918976, -0.25036025047302246, -0.3699646294116974, 0.7310354113578796, 2.800119161605835, 0.07702593505382538, -0.20561404526233673, 0.14736832678318024, 1.8876818418502808, -1.3941946029663086, -0.2956509292125702, 0.923608124256134, 1.0583903789520264, 0.57420814037323, 2.695422649383545, -0.3394915759563446, -1.0886143445968628, 0.8366336822509766, -0.9438250660896301, -0.7094441056251526, -0.42815718054771423, -1.7206813097000122, -0.3873080909252167, 0.42447394132614136, 1.1267708539962769, 0.3008256256580353, 0.4733610153198242, -0.3056583106517792, -0.2141030728816986, 0.5435968637466431, -1.6201497316360474, 0.08662857115268707, -1.7329092025756836, -0.7324503660202026, -0.6155475974082947, 0.18950174748897552, 0.9980229139328003, -0.33491820096969604, 1.1360411643981934, 1.4363996982574463, 0.01888260431587696, -0.8234128952026367, 0.6579676866531372, 1.3503371477127075, -0.22880811989307404, -1.3613765239715576, -0.9620174765586853, 0.26868560910224915, -0.949358344078064, -1.0598316192626953, 1.3244198560714722, 0.2724250555038452, -1.2262800931930542, 0.34266409277915955, 0.08115249127149582, -1.0223957300186157, 1.4953062534332275, -0.49643924832344055, 0.4205053448677063, 2.0557873249053955, 1.0966465473175049, -0.0943673625588417, -1.7434014081954956, 0.3609468340873718, 0.12440403550863266, -0.7471330761909485, 0.43590885400772095, -0.33374640345573425, 0.30829745531082153, 0.6927748322486877, -1.9618366956710815, 0.2074660360813141, 1.0120930671691895, 0.4256664216518402, 0.5117214918136597, -0.8701257705688477, 1.1270357370376587, -2.118225574493408, 1.9090583324432373, 1.7343640327453613, 0.3989609479904175, -0.17476709187030792, -1.5431747436523438, 0.9976247549057007, -2.114041328430176, -1.262985348701477, -0.04456634819507599, -1.3511487245559692, -0.08273892104625702, -0.4861101508140564, -0.808742344379425, -1.8546379804611206, -0.14587251842021942, -2.1777188777923584, 0.6595386266708374, 1.730244755744934, 2.909856081008911, 1.3783869743347168, -0.2545805871486664, 1.5804543495178223, 0.7347885966300964, -0.593633770942688, 0.901226818561554, 1.1548182964324951, 1.0018343925476074, -0.24103914201259613, -0.35071125626564026, 0.8351936340332031, -0.7208588719367981, 0.8430769443511963, -0.005066163372248411, 0.19752293825149536, -0.5296363830566406, 0.0032908266875892878, -1.204573392868042, 1.6148604154586792, 0.6113805174827576, 0.19388923048973083, -0.9903202056884766, -1.3398950099945068, 1.2860337495803833, 1.1885453462600708, 2.037187337875366, 0.5457533597946167, 0.9952631592750549, 0.19734716415405273, -0.14099697768688202, -0.4363144040107727, -0.8531665205955505, -1.202831745147705, -0.4079083800315857, -0.6052460670471191, -0.7314119935035706, 0.18229851126670837, -0.7795364856719971, 0.08245382457971573, -0.3942376673221588, -0.7849732637405396, 1.714147925376892, -1.2022441625595093, 2.2512590885162354, 0.1523076742887497, 0.03785860538482666, -0.7898285984992981, -0.6338319778442383, 0.6782466769218445, 0.6990919709205627, -0.4794919192790985, -1.341995358467102, 0.580579400062561, -0.5540223121643066, -2.0570170879364014, -0.398629367351532, -1.848624587059021, -0.6115332841873169, 0.9496694207191467, -0.10849296301603317, -0.9125361442565918, -0.6954203248023987, -0.5748893022537231, -1.074980616569519, -0.7103851437568665, 0.2807522714138031, -1.1704529523849487, -0.4622240662574768, 2.1542437076568604, -1.1760398149490356, -1.198507308959961, 0.09681101888418198, 0.2702294588088989, 0.6464745998382568, -0.8742675185203552, -0.8134316205978394, 0.5247296094894409, -0.5692096948623657, -0.3261949419975281, 1.009336233139038, -2.239351749420166, -0.8618956804275513, 0.3451116979122162, -0.9175068736076355, 1.4613397121429443, 1.2579030990600586, -0.707349956035614, 1.770710825920105, -1.047782301902771, 0.5487668514251709, -0.8320565223693848, -0.6289617419242859, -2.847372531890869, -0.8336183428764343, 0.29041728377342224, -0.998006284236908, 0.6079445481300354, 1.352919101715088, -1.1062368154525757, 0.7389329075813293, 0.9962781071662903, -1.6625797748565674, 2.0881905555725098, -0.23266950249671936, 1.0430145263671875, -0.6152447462081909, 0.26997604966163635, 1.7610942125320435, 1.7790433168411255, -0.358181357383728, -2.1690242290496826, 1.9971095323562622, -0.10957187414169312, 0.6525733470916748, 0.24477753043174744, -0.4871770143508911, 0.14237336814403534, -0.792589545249939, -0.4774971902370453, 0.41929173469543457, 2.3000030517578125, 1.3928215503692627, -0.7827436327934265, 1.178589940071106, 1.1472139358520508, -0.4944610595703125, 0.8094126582145691, 1.001876950263977, -1.5845179557800293, -0.1500338464975357, 0.8028026819229126, -0.5766881108283997, 0.7924139499664307, -0.14565309882164001, 0.43822240829467773, 0.266008198261261, -0.16277901828289032, -1.159085750579834, 0.7169427275657654, -0.23791468143463135, 0.5940089225769043, 0.47387975454330444, 1.0946917533874512, -0.11858547478914261, -0.9411816000938416, -0.7582348585128784, 0.7816972136497498, -0.5592133402824402, -1.332950472831726, -0.23720881342887878, -0.009191667661070824, -0.7296606302261353, 0.206920325756073, 0.010284285061061382, -0.3962138593196869, 0.6346575617790222, 0.8961395025253296, -0.11595764756202698, 0.21808503568172455, 0.3227558434009552, 0.773543119430542, 0.23658175766468048, -0.9873776435852051, 0.7936912775039673, -0.6522831916809082, 0.5072779655456543, 0.44026678800582886, -1.2768316268920898, 0.562201738357544, 1.6446021795272827, -0.2645638585090637, -0.7661062479019165, 2.1877527236938477, -0.8435478210449219, 0.0416841059923172, 0.31429219245910645, 0.4694713354110718, -1.5862247943878174, -0.7800422310829163, -1.000504970550537, -0.9782578945159912, -0.809253454208374, -0.10932882875204086, 0.057981349527835846, 0.00854989979416132, -0.10684725642204285, 1.4322376251220703, -0.5027066469192505, 0.15961384773254395, -1.017897129058838, 0.6637341380119324, 0.43460291624069214, -0.3834354281425476, -0.6370282769203186, 0.058273863047361374, 0.8605793118476868, 0.38361936807632446, 0.8464815020561218, 0.23176561295986176, -0.699870228767395, 0.06130828708410263], [1.7916216850280762, -1.7470672130584717, -0.0907275378704071, -1.0159635543823242, -0.19399185478687286, -1.7335277795791626, 0.30444860458374023, -0.8857550024986267, -0.08503399789333344, -0.1318129599094391, 2.1691434383392334, -0.7738947868347168, 0.6649753451347351, -0.5163207650184631, 0.23024410009384155, 0.8734697103500366, 0.06508919596672058, 1.378550410270691, -2.299598455429077, -1.0060086250305176, 1.893790602684021, -0.9086087942123413, -0.0381416454911232, -0.24235326051712036, -1.0131725072860718, -0.3526211082935333, 0.7552173733711243, -0.27733612060546875, 0.10307653993368149, 0.34993770718574524, 1.269500732421875, -0.30901750922203064, -1.300601601600647, -2.005185842514038, -1.3172801733016968, 0.745219886302948, -1.5804725885391235, -0.22743414342403412, -0.4940987229347229, -0.0736643597483635, 0.8415786027908325, -0.3684247136116028, 0.536300003528595, 1.5919404029846191, 1.1555594205856323, -0.7335773706436157, -0.39569687843322754, -1.1144187450408936, 0.7822853326797485, 1.160722017288208, 0.6222314834594727, 0.129899799823761, -0.6634098291397095, 1.450467824935913, 0.10239578038454056, 0.6092705726623535, -1.377784252166748, 0.6611093878746033, -0.8636149764060974, 0.3646232485771179, -0.14672774076461792, 0.38581186532974243, 1.9908424615859985, 1.529321312904358, -0.4055204689502716, 0.39554986357688904, 0.8405433297157288, 0.5246020555496216, -0.06800034642219543, -0.029999595135450363, -0.6898358464241028, 0.614106297492981, -1.418518304824829, 0.3823411464691162, 0.1352795660495758, -0.8818716406822205, -1.9267230033874512, 1.0086013078689575, 0.15491318702697754, -0.9507877230644226, -0.4843433201313019, 3.3408524990081787, -0.3139974772930145, -0.1914970725774765, 0.742516040802002, 1.7754172086715698, -1.194554328918457, -0.8639535903930664, 1.5480626821517944, -0.9870728850364685, -0.08656838536262512, -1.772824764251709, 0.9715722799301147, -1.6154428720474243, -0.8773495554924011, 0.5563469529151917, -0.3298492729663849, -1.2764886617660522, -1.1960054636001587, -0.38215580582618713, 0.4833639860153198, 0.8865906596183777, 0.9057595133781433, 1.0274661779403687, -1.3456205129623413, 0.5770888328552246, 0.7940717935562134, 1.009162187576294, -2.0011966228485107, 1.2794065475463867, 0.06607680022716522, 0.6042091250419617, 0.6256288290023804, 0.9441774487495422, -2.2216877937316895, -1.0601686239242554, -1.4702924489974976, -1.6802704334259033, -1.0303072929382324, -0.7161456346511841, 0.8488382697105408, -0.26352575421333313, -0.02768941968679428, -0.1666506975889206, -0.40762990713119507, -0.9063718318939209, 2.8592605590820312, 1.2180277109146118, 1.8374700546264648, -0.4390515983104706, 0.6845033764839172, 0.8086464405059814, -0.7056169509887695, -0.5201014876365662, -0.6934606432914734, -0.005131865851581097, 0.5300103425979614, 0.6802627444267273, 0.23695287108421326, 0.470495343208313, -1.20847487449646, 0.9260613322257996, 0.4439813494682312, 0.9741901159286499, 0.6507636308670044, 1.5332155227661133, -0.88340824842453, 1.0860387086868286, -1.9783216714859009, 1.37893807888031, 1.4243237972259521, -0.25131112337112427, -2.7872567176818848, 1.0161811113357544, 1.0678744316101074, 0.12214881926774979, -0.15792478621006012, -1.1866884231567383, -0.11785956472158432, -0.09292823821306229, 1.781126618385315, 0.7578011155128479, 1.5592767000198364, -0.25281664729118347, -0.886677622795105, 0.6590874791145325, -0.27570831775665283, -0.28288784623146057, 1.5295922756195068, -1.1949355602264404, 1.1050872802734375, -1.7683758735656738, 0.2482837438583374, 0.42799147963523865, -0.22867494821548462, 0.9536285400390625, 0.1875818818807602, 0.8067862391471863, 0.43098992109298706, -0.5489158630371094, 0.14334696531295776, -0.8240373134613037, 0.25711768865585327, -1.6820316314697266, 0.7345723509788513, 0.6419970393180847, -0.8763478994369507, -0.30544179677963257, 1.1542283296585083, 0.2591206729412079, -0.02911829575896263, -0.12170369178056717, 1.3484985828399658, 1.363325834274292, 0.21937255561351776, 1.5097885131835938, 0.8223209977149963, 0.40872395038604736, -0.3279463052749634, 1.8703571557998657, 1.3597419261932373, -0.4052344858646393, 1.0519604682922363, -0.8404978513717651, -0.4109846353530884, 0.8331004977226257, 0.12621448934078217, -0.4327307939529419, -0.9989751577377319, -1.9516688585281372, 0.0847359150648117, -1.4252293109893799, 0.8564444184303284, -0.12189232558012009, -0.1670088768005371, -0.9531964063644409, -0.4115978479385376, -0.9395568370819092, 1.9429993629455566, -0.009281362406909466, 0.23360781371593475, 0.2507137060165405, 0.39407652616500854, 0.7370175123214722, -1.2963885068893433, 0.2556169033050537, -0.778677225112915, -1.0722028017044067, 0.8710360527038574, -1.5664154291152954, -1.535300374031067, -1.2980318069458008, -1.2056763172149658, 0.2688828408718109, -0.06111977994441986, 1.1504285335540771, 1.5687665939331055, 1.8887834548950195, -1.676398515701294, -0.6030126810073853, -0.9534406065940857, -0.18478025496006012, 1.0572876930236816, -0.9548019766807556, 0.7535445690155029, -0.23367814719676971, 0.2957518994808197, 0.23144608736038208, -0.3032526969909668, -1.605080008506775, -0.029364824295043945, 0.40080636739730835, -1.2600624561309814, -1.7867153882980347, 0.7004861235618591, 2.1020009517669678, -1.0815925598144531, -0.6134129762649536, 0.7733556032180786, 1.333721399307251, -0.7838674783706665, -0.8205838203430176, 1.3694267272949219, -1.3877806663513184, -0.7941301465034485, 0.31696873903274536, -1.1842883825302124, 0.9927252531051636, -0.8950533866882324, -0.8869236707687378, -0.1772446483373642, -0.9334468841552734, 0.1221669539809227, -0.2790684700012207, -0.9745341539382935, 0.018886476755142212, -0.22406499087810516, -0.822404146194458, 0.819573700428009, -0.921979546546936, 0.1702534556388855, 0.22721488773822784, 1.4433737993240356, -0.7467254996299744, 0.6220235228538513, -0.31060606241226196, 1.9351822137832642, 0.4086562693119049, -0.14767475426197052, -0.101804219186306, 0.7738782167434692, 1.1651023626327515, 0.7384998798370361, -1.2865320444107056, -1.0460386276245117, -1.4272665977478027, -0.8958088159561157, -1.7164491415023804, 0.6258264183998108, -0.7225490212440491], [1.5664781332015991, 0.1253211796283722, 0.7499346733093262, -0.2703284025192261, 0.3654044568538666, 0.5186241865158081, 1.0202357769012451, 0.5387547016143799, -0.8820221424102783, 0.8065973520278931, 1.668721079826355, -1.8438652753829956, -0.30613380670547485, 0.1694834977388382, -0.9713944792747498, -0.6656961441040039, 0.8473515510559082, 0.7440451383590698, -1.6919691562652588, 0.13064900040626526, -1.0091701745986938, 1.1196579933166504, -2.255582332611084, 0.07476618885993958, 0.1426948606967926, 1.1269152164459229, 0.24283696711063385, 1.3250776529312134, 0.6906304359436035, 1.0298221111297607, -0.45642217993736267, -0.01271410658955574, 0.4325287938117981, 0.20501355826854706, -0.23191700875759125, -0.02077794075012207, 1.5349504947662354, 0.5566859245300293, 1.0015950202941895, -2.4144303798675537, -0.8392035365104675, -1.2683932781219482, 0.3712035119533539, -1.4049991369247437, -0.974663496017456, 0.09985307604074478, 0.48072192072868347, 0.13873209059238434, -1.5385475158691406, 0.9029324650764465, -1.2029163837432861, 1.2217518091201782, -1.0467708110809326, -0.0029897650238126516, 1.0083316564559937, 0.38279780745506287, -0.6635498404502869, 1.2663707733154297, -0.28366777300834656, 0.08348789811134338, 0.6235314607620239, 0.13956299424171448, 0.4463783800601959, 0.3990284204483032, -0.636647641658783, -0.8430559039115906, -0.9852579832077026, 1.2502543926239014, 1.1625425815582275, 0.49433454871177673, -0.261827677488327, -0.9704712629318237, 1.7058414220809937, -0.7024155855178833, 1.8507447242736816, -0.12423136085271835, 0.42821159958839417, 1.6311124563217163, -0.2802153527736664, 0.6099098324775696, 0.9501694440841675, 0.5139588117599487, -0.5103549361228943, -1.7761861085891724, 0.594202995300293, 0.046945538371801376, 1.1190879344940186, -0.9836484789848328, 0.7200968861579895, 0.4213986098766327, -1.784716010093689, -0.9395902156829834, 1.74650239944458, -1.379607915878296, -0.5510604977607727, 3.0077717304229736, -1.2967201471328735, -0.9643728733062744, 0.4952748715877533, -0.17538709938526154, -0.6671570539474487, 0.9402768015861511, -0.04477451741695404, -0.7046225070953369, 1.2163293361663818, -2.692931652069092, 0.8912607431411743, -0.48715677857398987, 1.0948970317840576, 2.275596857070923, -0.20014789700508118, -1.183090090751648, 0.8599424362182617, -1.1634422540664673, 0.598048746585846, 0.3312647342681885, -0.27208641171455383, -0.7994574308395386, 0.2534060776233673, 0.4363572895526886, 0.7691405415534973, -0.00645790109410882, 0.3886958956718445, 1.3483550548553467, -0.7966557741165161, 0.9643183946609497, 0.5461503863334656, 0.9585481882095337, 0.6087843775749207, 0.7195262312889099, -0.16335806250572205, -0.14230260252952576, 0.19832947850227356, 0.3716401755809784, -0.12543265521526337, -1.9677727222442627, -0.08029954880475998, -0.9393415451049805, -0.5960623025894165, -0.16826744377613068, -1.2337878942489624, 0.5211378335952759, 1.1254611015319824, 1.2150815725326538, -0.06704255938529968, 1.0544402599334717, -1.2136050462722778, 1.0474220514297485, 0.08604367077350616, -1.3377867937088013, -0.5307633876800537, 1.5018367767333984, 0.4291449189186096, 2.872965097427368, -0.6459458470344543, 0.0671345666050911, -1.2517445087432861, 1.3055537939071655, 1.0040680170059204, 1.15602445602417, -0.40104568004608154, -0.3903028070926666, -0.7681064009666443, -1.0089298486709595, 0.2289709448814392, -0.9789212942123413, -1.401840090751648, -1.4050321578979492, -0.8406880497932434, 1.8005050420761108, 0.8519758582115173, 1.4861725568771362, -0.9035146236419678, 0.7884764075279236, 0.43301627039909363, -2.567493438720703, -1.1336021423339844, -0.6033223271369934, -0.875698983669281, 0.9608855843544006, 1.874804973602295, 0.57854825258255, -1.0889923572540283, -0.354642778635025, -0.20040671527385712, 0.307033509016037, 0.8048043251037598, -0.7237367033958435, -0.6551392674446106, -0.2544199824333191, 0.18491436541080475, 1.8167484998703003, 0.818562924861908, -0.7809691429138184, 0.5641648769378662, -0.47398558259010315, 1.2086851596832275, 1.0360842943191528, 1.8124210834503174, 1.0692051649093628, -0.3601837456226349, -1.0271097421646118, -0.0715113952755928, -0.7141872644424438, -0.6456541419029236, -0.7554125785827637, -1.4188181161880493, -1.9504377841949463, 0.6439616680145264, 2.0116114616394043, 1.9207366704940796, -0.7225656509399414, -1.232534408569336, 1.763475775718689, 1.1686663627624512, -0.7463586926460266, 0.2972547113895416, -0.265415757894516, 0.28181979060173035, -0.11923308670520782, -0.5748851895332336, 0.11652936786413193, 1.659619688987732, 1.9533061981201172, 0.053793031722307205, -0.7834741473197937, 0.9431117177009583, 0.8272011280059814, 1.1961153745651245, 1.0258252620697021, 2.3095009326934814, 0.04836248606443405, -0.7814629673957825, 0.18339364230632782, -1.4827427864074707, 0.9831149578094482, 0.4562666118144989, -1.4619417190551758, 0.33819934725761414, 0.5789136290550232, -0.3618254065513611, -0.03999623283743858, -1.247638463973999, -0.17132721841335297, 1.6258872747421265, -1.4106122255325317, 0.876473605632782, 0.07834542542695999, -2.166396141052246, 0.5792449116706848, 1.6263024806976318, 0.5009849667549133, -1.4512561559677124, -0.3948354721069336, 0.04361405968666077, -0.04132703319191933, 0.4887447953224182, 0.07263670116662979, 1.0206298828125, 0.9560257792472839, 1.606994867324829, 0.2886609733104706, 1.2270768880844116, 0.12734580039978027, -0.9072296619415283, 0.4270804822444916, -0.5182508230209351, -0.6923123598098755, 0.42966923117637634, 1.5242931842803955, 0.49702683091163635, -0.4248661696910858, 0.708228588104248, 1.243760585784912, -2.000993013381958, 0.5141034722328186, 2.017651319503784, -0.981728732585907, -0.6866782307624817, 1.7224104404449463, 0.7446081042289734, -1.1408780813217163, 1.6220930814743042, 0.9634043574333191, 0.4526364505290985, 0.6478329300880432, -1.3920453786849976, 0.5433608293533325, -0.7388334274291992, 0.6101140379905701, 0.4146769344806671, 1.4027539491653442, -1.1808775663375854, -0.6464345455169678, -0.20661884546279907, 0.4805504083633423, 2.166090488433838, 0.11208227276802063, -1.3223220109939575, 2.30450701713562], [-1.830290675163269, -0.5031300783157349, 0.4185294210910797, 0.25635474920272827, 1.4754325151443481, 1.6412073373794556, -0.08492133766412735, -0.4991627037525177, -0.5132540464401245, -0.45375898480415344, -0.08807212114334106, 0.1988806575536728, -0.0004061048093717545, -1.7364906072616577, -0.57113116979599, 0.9440582990646362, -0.43794286251068115, 0.17174264788627625, -0.30547282099723816, 0.46878254413604736, 1.064146876335144, 0.1325756162405014, -0.9101865291595459, 0.20628179609775543, 0.16298651695251465, -1.0062105655670166, 0.05549902468919754, -0.26209819316864014, -2.9451065063476562, 0.27983635663986206, 0.704937756061554, -0.4768337905406952, 0.11907301098108292, -0.3780675232410431, -0.8794834017753601, 0.36143946647644043, -0.3957262933254242, 0.12212063372135162, 0.21034519374370575, 1.012649655342102, -0.9735077023506165, 0.3154914081096649, 0.6269667148590088, 2.835632085800171, 1.179195761680603, 0.9611049890518188, 1.0929909944534302, -0.5661680698394775, 1.890838623046875, 0.7696443200111389, 0.21549731492996216, -0.0858127772808075, 0.892778754234314, -0.5524889826774597, 0.26880329847335815, 1.1896625757217407, 1.0119906663894653, 0.12436889857053757, 1.5415043830871582, -2.4829680919647217, -0.6846128106117249, 1.155958890914917, 0.2533941864967346, -0.6469513177871704, -0.14730338752269745, 0.9076643586158752, -0.23375900089740753, -0.07643889635801315, 1.253345251083374, 0.8750450611114502, -0.3849937319755554, -1.3849766254425049, 0.3643278479576111, 0.4035418629646301, -0.7381467819213867, -0.2719842493534088, -0.1327238231897354, -1.5776422023773193, -0.416096955537796, -0.4485756754875183, -0.07308768481016159, -0.8255351185798645, -1.8998057842254639, 0.5343860983848572, 0.8565766215324402, 0.8311083316802979, 1.3208328485488892, -1.67227041721344, 0.5209677815437317, 0.8117272257804871, -0.8571081757545471, -0.6007856130599976, 0.973185658454895, -0.9153668284416199, -0.8063853979110718, 0.5997666120529175, -1.1042194366455078, 0.2965173125267029, -0.7612497210502625, 0.9291250109672546, -0.7445635795593262, -0.9713926315307617, 1.6470146179199219, 0.4815712869167328, 0.9523417353630066, -1.6522128582000732, 0.897760272026062, 0.8971235752105713, -0.3702701926231384, -2.8532052040100098, -0.08676164597272873, 1.7510969638824463, -1.2187479734420776, 0.42664456367492676, -0.6222152709960938, 0.959413468837738, -1.0730794668197632, -1.7641927003860474, -0.4717186391353607, -0.20278948545455933, 0.7782483100891113, 0.6740607619285583, 0.9423393607139587, 0.6605913639068604, 0.9094329476356506, -0.5516421794891357, 1.269083023071289, -0.7402815818786621, 1.232292652130127, -1.3780444860458374, -0.32786184549331665, -2.4503157138824463, -1.3391146659851074, -2.115018844604492, -0.11228715628385544, 1.6588839292526245, -2.151563882827759, 1.1377100944519043, -1.907682180404663, 0.05839379131793976, 0.1574859619140625, 1.0744435787200928, -0.1393151879310608, -0.3871631920337677, 0.3464101552963257, 1.415972113609314, -1.779007911682129, -0.6929082274436951, -1.3330378532409668, -0.07005979120731354, -0.43246880173683167, -0.675116240978241, -1.1265742778778076, 0.25704944133758545, 0.4488535523414612, 0.6461544632911682, 0.4184347093105316, 1.0416027307510376, 0.6028217077255249, 0.555955171585083, 0.19801677763462067, 1.0138736963272095, -1.0805397033691406, -0.358569473028183, 0.3255103826522827, -0.5721645951271057, -1.0968897342681885, 0.38319867849349976, -0.22723433375358582, -1.3672102689743042, 1.5368695259094238, 0.1937081217765808, 0.2959974408149719, 0.37212449312210083, 0.4366133213043213, 0.19687393307685852, 2.0076353549957275, -1.872222900390625, 0.45698627829551697, 0.760495662689209, -1.0157983303070068, 0.05779644474387169, 1.0419178009033203, -0.4101136028766632, 0.49919694662094116, -0.16155840456485748, -0.7026262879371643, -0.7207171320915222, -0.5463602542877197, -0.29047727584838867, 1.0921279191970825, -0.8550440073013306, 1.2067666053771973, -0.46251675486564636, -0.8116133809089661, -1.1656506061553955, -0.033218223601579666, 0.1065925806760788, 0.11999129503965378, -1.0818779468536377, -0.359531968832016, -0.5672540068626404, 0.11452030390501022, 0.9627607464790344, -1.1404671669006348, 0.7451988458633423, -1.236499547958374, 0.6717681884765625, -1.0075844526290894, 0.2548743784427643, 0.11514884233474731, 1.8643828630447388, 0.15411309897899628, 0.7441577315330505, 0.5908025503158569, 0.7246561050415039, -0.37955209612846375, -0.5032054781913757, 0.21415632963180542, -0.2469160556793213, 2.625763416290283, -0.5555692911148071, 1.7143553495407104, -0.9846976399421692, -0.9944050312042236, -0.6103855967521667, 0.041191622614860535, 0.44903072714805603, 1.2528964281082153, 0.08483874052762985, 0.9922690987586975, -0.3681836724281311, -1.4886709451675415, -0.9008527398109436, 0.57049959897995, 0.1456974744796753, 0.5321523547172546, -0.7035025358200073, -1.0593072175979614, 0.9624011516571045, 0.8044664263725281, 1.2193477153778076, 0.5653429627418518, 0.8128646612167358, 0.8897463083267212, -0.47405022382736206, -0.06883133202791214, -0.5722438097000122, -0.5463438630104065, 0.6165976524353027, 1.0324383974075317, -1.3223152160644531, 2.2221245765686035, 1.7140446901321411, 0.6680114269256592, -1.5570918321609497, 0.2005963921546936, -0.16588200628757477, 0.41689741611480713, 0.6378927230834961, 0.3277653157711029, -0.3349679112434387, -0.26714667677879333, -1.522919774055481, -0.9604791402816772, 0.07928374409675598, 0.10095608979463577, 0.7290504574775696, -0.08136701583862305, -1.3102778196334839, -1.0050443410873413, 0.7489272356033325, 0.09870810806751251, -0.3667471706867218, -0.37427201867103577, -0.7921085357666016, -0.9389812350273132, 0.03994214907288551, 0.11329881101846695, 0.24569043517112732, 0.9776369333267212, 1.1837602853775024, 0.052485108375549316, 0.6433362364768982, 0.16050748527050018, 0.8441039323806763, -0.08288485556840897, 0.4487014710903168, -0.9095426797866821, -0.297558069229126, -1.9805543422698975, -0.011951006948947906, 2.488600254058838, -0.05133848264813423, 1.3109160661697388, -1.0492289066314697, -1.2369463443756104, -0.7919847369194031, -0.6559720635414124, 1.266422152519226], [-0.8906143307685852, -0.5833732485771179, -0.8526157736778259, -0.6098753809928894, 0.9803419709205627, 0.8611692786216736, 2.214740037918091, -0.6531839370727539, -0.6651816368103027, -0.47594913840293884, 0.16962474584579468, 2.330554485321045, -1.8673287630081177, -0.9280169010162354, 0.4331531226634979, -1.2801437377929688, 0.013273019343614578, -0.8873732686042786, 1.161385416984558, -0.15978457033634186, -0.4387434422969818, 1.8924157619476318, -0.3007793724536896, -0.7899206280708313, -0.034834735095500946, 0.2357240915298462, -0.17730143666267395, -0.5485159754753113, -1.434162974357605, 0.8769750595092773, -0.7543386220932007, 0.42597582936286926, 1.0464739799499512, 0.3064541816711426, -2.164691925048828, 0.1573718935251236, 0.797479510307312, -0.13573956489562988, 0.16535072028636932, -0.2853359878063202, 0.3445444107055664, -1.2041258811950684, 0.7272298336029053, -0.6139771342277527, 1.3295762538909912, 0.41927722096443176, -0.3234463036060333, 1.3317259550094604, -0.7971570491790771, 0.46350082755088806, -0.9171501994132996, 0.4545232355594635, 1.0842634439468384, -0.45792123675346375, -1.660815954208374, -0.5251814723014832, 0.6950910091400146, 1.2040499448776245, -0.023132847622036934, 0.9105571508407593, -0.1451536864042282, 1.766383171081543, -0.20487061142921448, 0.49651262164115906, 0.7774478197097778, 0.9508094787597656, -0.13517293334007263, 1.4073513746261597, -0.9385727047920227, 1.1681413650512695, 1.4656155109405518, 0.8169611692428589, 1.3672611713409424, -0.7396061420440674, 1.4319490194320679, 1.18778657913208, 1.300770878791809, 0.9020026922225952, 0.36042192578315735, -1.180019736289978, 0.9954758882522583, -0.36779290437698364, -0.6457930207252502, -0.8035629391670227, -0.490899920463562, 0.6131772994995117, -0.586126446723938, 1.2090647220611572, -0.9892812371253967, -0.4852021038532257, 0.4157886505126953, 0.7186562418937683, 0.9176943302154541, -0.3306959867477417, 0.224701389670372, -0.4083988070487976, -1.5403625965118408, -1.2076455354690552, 2.0857603549957275, -1.7881921529769897, 0.5123338103294373, -0.6671865582466125, -1.109678864479065, -0.199799582362175, -1.6746569871902466, 1.0661475658416748, 1.2751282453536987, 0.23651361465454102, 0.8670586943626404, -0.7675366401672363, -1.0481276512145996, -1.2301465272903442, -0.4114125669002533, -1.83534836769104, 0.6401938199996948, -0.08487044274806976, -2.1029305458068848, -1.3125557899475098, -1.943949580192566, 0.26425325870513916, 0.15559875965118408, 0.3145202696323395, -0.09047747403383255, -0.18964846432209015, -0.3388117551803589, -1.4074742794036865, -0.11199625581502914, -0.22107580304145813, 1.3349624872207642, -0.47043681144714355, -0.9335514307022095, -1.4723573923110962, -0.6639293432235718, -0.6681101322174072, 0.1781385838985443, -0.03495948389172554, -0.7125821113586426, -0.9686612486839294, 0.8459296822547913, 1.1723120212554932, -0.9901268482208252, 1.8766756057739258, 0.26462969183921814, -0.9601462483406067, 0.7661184072494507, -0.285637766122818, -2.485898494720459, -0.026729334145784378, -0.7812440991401672, 1.1437093019485474, 0.4091019034385681, -1.6564122438430786, 0.1950441300868988, -1.384480595588684, 1.6948117017745972, -0.03155237436294556, -0.4584179222583771, 0.3926577568054199, 0.21065400540828705, -0.6991976499557495, -0.6269458532333374, -1.0975593328475952, 0.14398251473903656, -0.055497895926237106, 0.599571704864502, -1.8305635452270508, -0.40668991208076477, -0.5744528770446777, -0.177459716796875, 0.8169228434562683, 0.5153211355209351, -1.1045475006103516, -0.4886810779571533, 1.624763011932373, -0.12575840950012207, -1.2259321212768555, 1.996538758277893, 0.2100904881954193, -0.17171910405158997, -0.7386049628257751, 0.1909700483083725, 0.7136655449867249, 2.098170042037964, 0.7693222761154175, -0.5342634916305542, -0.16385379433631897, 0.6189790964126587, 0.014471742324531078, 0.3302956819534302, 0.914421021938324, 0.5545591115951538, -2.1379895210266113, 0.45778217911720276, -1.4658766984939575, -1.5596579313278198, -0.17254339158535004, -0.6774942874908447, -1.3550828695297241, 1.0388610363006592, -0.08457565307617188, -2.284219741821289, 0.7417581081390381, 1.2289950847625732, -0.20360083878040314, 0.5259379148483276, -0.6909196376800537, 0.9567028284072876, 0.9794175624847412, 1.3778321743011475, -1.071323037147522, -0.6081165671348572, 0.6425497531890869, 1.1058909893035889, 1.0203194618225098, 1.464882254600525, 1.4163532257080078, 0.6907607913017273, -0.6694486737251282, -0.6605199575424194, 1.1143555641174316, -0.0172817911952734, 1.0420454740524292, -1.1892403364181519, -1.0382096767425537, 0.6424106955528259, -1.271520972251892, 0.2303183674812317, -0.22409677505493164, -2.0879716873168945, -1.161062479019165, 0.3116496503353119, -0.29760080575942993, -1.7501753568649292, 1.5534790754318237, -2.2280220985412598, 0.17033274471759796, -1.0033589601516724, -0.3147470951080322, -0.9777894020080566, 1.0169932842254639, 1.315722942352295, -0.8611148595809937, 0.31305673718452454, -0.3674561083316803, -0.7949886322021484, -0.6240066289901733, 0.9187471866607666, 0.4007817804813385, -0.8857910633087158, 0.3583301901817322, -0.6771237254142761, -1.452376365661621, 0.14885488152503967, 1.1959470510482788, 1.76726233959198, 0.7595295310020447, -1.1691415309906006, 0.16265006363391876, -0.7382741570472717, -1.0689857006072998, 1.1271065473556519, -0.9996855854988098, -1.938136339187622, 1.0669190883636475, -1.4413520097732544, -0.05564635246992111, -0.7177651524543762, 3.1402130126953125, -0.04071299731731415, -0.9851672053337097, -0.8983857035636902, 0.9175973534584045, -0.5983770489692688, 1.5197925567626953, -0.07361792773008347, -1.1543035507202148, -0.01959393545985222, 0.43700578808784485, 1.0252690315246582, -0.024066969752311707, -1.2027294635772705, 0.48609021306037903, -0.11114539206027985, 0.5900579690933228, 0.4450553357601166, 0.7516725659370422, -0.718639612197876, -0.03029598668217659, -0.04666285216808319, 0.46414127945899963, 0.4203968942165375, 0.5901908874511719, 0.8559447526931763, 0.5607712864875793, 1.1746982336044312, 0.42892006039619446, 0.7059705257415771, 0.09105350822210312, -1.2438045740127563, -0.8749289512634277], [0.44599273800849915, 1.123768925666809, 1.3359453678131104, 1.0709068775177002, 0.30660781264305115, 1.9282119274139404, -0.9169585108757019, 1.1153044700622559, -0.6048136949539185, -1.0075665712356567, 0.6120656132698059, -1.0483049154281616, -0.20397867262363434, -1.6248763799667358, 1.8809409141540527, 1.3671600818634033, 0.31386858224868774, -0.10706377029418945, -0.016634562984108925, 1.4149348735809326, -1.1537017822265625, 0.09999285638332367, 1.5815187692642212, -0.15423783659934998, -0.3340660631656647, -1.1821397542953491, 0.8977771401405334, 0.053973183035850525, 0.4384632706642151, -1.1752558946609497, 0.4595990777015686, 1.3096537590026855, -0.7325342893600464, 1.0721021890640259, 0.007540532387793064, -1.771825909614563, 0.1911337524652481, -0.10529295355081558, -0.7220704555511475, 0.4848085045814514, -1.3925164937973022, 0.6502612829208374, 0.242909774184227, -0.24723659455776215, -1.2777279615402222, -0.997448205947876, 0.6942363977432251, -0.14875011146068573, 0.07283788919448853, 0.11139438301324844, -0.9270625710487366, 0.3959093987941742, 0.12532784044742584, 1.0709415674209595, -0.8441274762153625, 0.024847332388162613, -1.3204611539840698, 0.20657609403133392, -0.47969135642051697, 1.3084957599639893, -0.3046075999736786, -0.16737642884254456, 0.1236599013209343, 0.49666351079940796, 0.6997963190078735, 0.8782932758331299, -0.6685040593147278, -0.4232544004917145, 1.2835924625396729, 0.386018306016922, 0.2710094749927521, -0.28994297981262207, 1.4606081247329712, 0.6465616226196289, -1.064168095588684, 0.5913880467414856, 0.49175122380256653, -0.9325580596923828, 0.7896860241889954, 0.9098542332649231, -1.4443366527557373, 1.1941449642181396, 1.0260127782821655, -1.1666768789291382, 2.072648763656616, -0.347552627325058, -0.3084505796432495, -0.19871176779270172, -0.21582719683647156, 0.4598836600780487, 0.39109376072883606, -0.47626960277557373, -1.6647148132324219, -0.4940108358860016, 0.9452401399612427, 2.2770884037017822, 1.1098028421401978, 0.4938846528530121, -0.46898353099823, -0.20078767836093903, -0.7603141069412231, 0.2590225636959076, 0.1376073807477951, -1.7780382633209229, 0.20585997402668, -1.1201146841049194, -0.2784603536128998, -0.7211665511131287, 0.8205970525741577, -0.15555769205093384, 0.16049890220165253, -0.26408424973487854, -0.40784406661987305, -0.8049497008323669, -0.28260743618011475, 0.8237696886062622, -0.43372830748558044, 0.33398690819740295, -0.9000943303108215, -1.439399003982544, -0.7015372514724731, -0.4516908824443817, -0.7234777808189392, -1.3593931198120117, 1.2676116228103638, 0.9981356263160706, -0.642334520816803, 1.1347506046295166, 0.048622164875268936, 0.8814626336097717, 0.1286081224679947, -0.0826977863907814, 0.23269571363925934, 0.5623148083686829, -0.17792214453220367, -1.2396503686904907, -0.9513311982154846, 2.1831445693969727, -0.782174825668335, 0.731605052947998, -0.2578120827674866, 1.0576269626617432, 0.8545525670051575, 0.015576308593153954, -1.0896607637405396, 0.891120195388794, -0.3685392737388611, 0.4751465320587158, 1.884648084640503, -0.21022281050682068, 0.4457074999809265, 0.456564724445343, -0.272389680147171, 0.7984153628349304, 1.1898212432861328, 0.10021983832120895, -0.17200665175914764, 0.2698950469493866, -0.413855642080307, 0.6078813076019287, 0.03476576879620552, 1.2556889057159424, -0.859880805015564, 1.2980728149414062, -0.35065510869026184, 0.5259090662002563, -0.7892055511474609, 0.8469299674034119, -1.3655189275741577, -0.026761861518025398, 0.38819947838783264, 0.1777707189321518, -1.252145528793335, 0.6044161915779114, 0.03928696736693382, 0.44189995527267456, 1.2858338356018066, -0.4378429651260376, 0.09700985997915268, 2.764174699783325, -1.3467172384262085, -0.5725651383399963, 0.18562866747379303, -0.9700056910514832, -1.615147590637207, 0.11002184450626373, 1.6664927005767822, 0.035637907683849335, -1.1822360754013062, -1.4955023527145386, -0.43286049365997314, 0.5637408494949341, 0.26376479864120483, -0.9541723728179932, -0.8580264449119568, 1.596135139465332, -0.7193084359169006, -0.21158352494239807, -0.38450974225997925, -0.25510647892951965, 0.16160692274570465, 0.1472209244966507, 0.07921592891216278, 1.3226025104522705, -1.089829921722412, 2.404911756515503, -2.0760834217071533, -0.6744959354400635, -1.1881159543991089, -0.12802253663539886, -0.7232416272163391, -0.029631296172738075, -1.2483506202697754, -0.33933088183403015, 1.0019110441207886, 1.0435763597488403, 0.6955034732818604, -0.03476342186331749, -0.8949488997459412, 0.5740018486976624, -1.339303731918335, -0.032040201127529144, -1.221880316734314, 0.8400803804397583, -1.3620326519012451, 0.5030941963195801, 0.04460981860756874, 0.8898813128471375, 0.32515084743499756, -0.35661691427230835, -1.111703872680664, 0.24576525390148163, -0.5603579878807068, -0.010110734030604362, -2.392392873764038, -0.5024220943450928, 0.2879670560359955, 0.7604702711105347, -0.29547348618507385, -1.822898268699646, 0.6578412652015686, 0.1734175831079483, -0.08434627950191498, -0.8047210574150085, 0.45215797424316406, -1.9272069931030273, -1.4583446979522705, -0.6282064318656921, 0.1356860101222992, 0.44738951325416565, -0.4968721270561218, -0.3298088014125824, -0.05991537496447563, 0.5061503052711487, 0.21116267144680023, -1.189200520515442, -1.4360114336013794, -0.2624287009239197, 0.033617258071899414, 0.7147479057312012, 0.1255752444267273, 1.0748059749603271, -0.2776601314544678, 1.2723066806793213, -0.9569398760795593, 0.34881895780563354, -0.3885899782180786, -0.5364113450050354, -0.9441208839416504, -0.5251978039741516, 1.8095669746398926, -0.5496580004692078, 1.1777188777923584, -1.4707272052764893, -0.04656150937080383, 0.8650758266448975, -1.5571961402893066, -0.156491219997406, -0.685298502445221, 0.13942255079746246, 0.7284310460090637, 0.3817957937717438, -1.392358422279358, -1.2590670585632324, -0.9215787649154663, -0.810425877571106, -1.5028029680252075, 0.6868105530738831, 1.2577829360961914, 1.0861196517944336, -0.4958645701408386, 2.0325896739959717, -2.151186227798462, 1.8649173974990845, -0.27292484045028687, -1.2335563898086548, -0.20450079441070557, -0.3342962861061096, -0.6303829550743103, 1.1086466312408447], [-0.6481336355209351, -0.37652331590652466, 1.0396809577941895, -0.6050899624824524, 0.6673210859298706, 1.7496980428695679, -0.8043507933616638, 0.8283720016479492, 0.45845136046409607, 1.1829428672790527, 0.5149032473564148, 1.1040287017822266, 0.24049076437950134, 0.4262528717517853, -1.6254738569259644, 1.512904405593872, -0.9616169333457947, 1.2592893838882446, 0.3959042727947235, -0.19455382227897644, 0.6033479571342468, -0.6766254901885986, -0.5867205262184143, -0.5513928532600403, -0.17528705298900604, 0.4468229115009308, -0.11675477027893066, 1.7425585985183716, -1.4561985731124878, -0.864752471446991, 1.3985635042190552, -0.7205846905708313, -0.014707690104842186, 1.5691072940826416, -0.14917173981666565, 0.3721420466899872, 0.33821019530296326, -0.2874602675437927, -2.570756673812866, 0.4940982460975647, 0.2647894024848938, -0.8235775828361511, 0.5411013960838318, 1.8500934839248657, -0.7113940715789795, -2.0764622688293457, 0.9859510064125061, -1.3357245922088623, -0.5656452775001526, 0.1251448690891266, -1.8098912239074707, -0.9924834966659546, -0.07088407129049301, 0.6252335906028748, -0.35478946566581726, -0.2086620330810547, -0.07771984487771988, 0.40308618545532227, -1.1294280290603638, 0.9661555290222168, -0.6476563811302185, 0.18640977144241333, 0.5334255695343018, 0.6353532075881958, -1.597328543663025, -0.07668357342481613, -1.7207472324371338, -0.10967351496219635, -1.3549754619598389, -0.44564443826675415, -0.2371155023574829, 0.7506181001663208, 0.8332710862159729, -0.6106032133102417, 1.5393520593643188, 0.8870759010314941, 1.1977157592773438, -0.7444832921028137, -0.5827844738960266, -0.32843953371047974, 1.6565214395523071, 0.41153666377067566, 0.9543840885162354, -1.1487380266189575, -1.2631784677505493, -0.1641692966222763, -0.625421404838562, -0.3960167169570923, -0.3353067636489868, 0.29120564460754395, 0.1114446222782135, 0.7539876103401184, -1.9601106643676758, -1.0688568353652954, 0.2671830952167511, -1.0637285709381104, 0.3345543444156647, 0.439942866563797, 0.19154120981693268, -1.2209926843643188, 0.5532903075218201, -0.5500365495681763, -1.0045641660690308, 0.045182403177022934, 0.6137642860412598, -0.5963295698165894, -0.5053239464759827, -0.9320284724235535, 0.92640620470047, 1.5074985027313232, 0.6180502772331238, -0.3226291835308075, 1.5848472118377686, 0.05425940081477165, 2.3522377014160156, -1.7703653573989868, -1.083947777748108, 0.013100836426019669, -0.13546255230903625, 1.2765629291534424, -0.9921949505805969, 0.8128540515899658, 0.4765559434890747, -0.644726574420929, -0.29605355858802795, -0.9160559177398682, -1.0410935878753662, -0.9286903738975525, 0.9036267399787903, -1.2633781433105469, 0.570033609867096, -0.6663624048233032, 1.3061039447784424, -1.4661297798156738, -0.21695709228515625, -0.4182911515235901, 0.28229787945747375, 0.38164788484573364, 1.3907544612884521, 0.8294857144355774, -1.111411690711975, -0.66154545545578, 0.46109578013420105, -1.6255712509155273, 0.41856881976127625, -1.0141425132751465, -0.8251355290412903, 0.30827826261520386, 0.12523368000984192, 0.9591553211212158, 1.9809973239898682, 1.5849851369857788, -1.5126285552978516, -0.7736878395080566, 0.783709704875946, -0.5529608726501465, 0.022169210016727448, -1.5388281345367432, -1.1526484489440918, -0.6092413067817688, 1.6423097848892212, -1.8265641927719116, 0.6862782835960388, 0.6487458348274231, -0.26331982016563416, -0.7447163462638855, -0.4553794264793396, -1.226378083229065, 0.1906461864709854, 0.5614212155342102, -0.608381986618042, 0.7270663976669312, -0.07495083659887314, 1.543210506439209, 0.2449764609336853, 1.9758187532424927, -1.5509225130081177, 2.1907198429107666, 1.4848421812057495, -0.3408370018005371, 0.32636478543281555, 1.9000499248504639, -0.4357428550720215, -1.3362510204315186, 0.32622864842414856, -1.3696037530899048, 0.9088364243507385, -0.27408072352409363, 0.5630316734313965, 0.6564155220985413, 0.2597334384918213, -0.369375079870224, -2.08298921585083, -0.43758854269981384, -1.3092446327209473, 0.12459686398506165, -1.291340708732605, -0.6201766133308411, 0.1255064755678177, -1.454534888267517, 0.05156456306576729, 0.6102274656295776, -0.20558641850948334, -1.1036063432693481, 1.299072265625, -0.10980800539255142, 1.3753411769866943, -0.06193138286471367, -0.8197519183158875, -1.5667484998703003, -1.2858409881591797, 0.5481421947479248, 0.9343869090080261, -1.196755290031433, 0.5326426029205322, -0.9714453220367432, 0.40784361958503723, 0.8739427328109741, 0.5480828285217285, -2.042942523956299, 0.9168223738670349, -1.319024920463562, -0.4024501144886017, 0.1844608187675476, -0.1990174651145935, -0.47288069128990173, -0.2606216073036194, 0.9459194540977478, 1.1619199514389038, -0.3811954855918884, 1.5760869979858398, 2.305377244949341, 1.694105625152588, -1.083768606185913, 0.5450522303581238, -0.12723296880722046, -0.04083015024662018, 0.20689266920089722, 0.35366910696029663, 0.48610278964042664, -1.0963151454925537, 1.4700453281402588, 1.4933655261993408, -1.5172628164291382, 1.0771745443344116, -0.32503876090049744, 0.42233988642692566, 0.05650552362203598, -1.067696452140808, -0.6606760025024414, -0.8839194178581238, -1.1286417245864868, -0.17015936970710754, -0.19066977500915527, 0.1296769082546234, 1.1149959564208984, -0.3212760388851166, 1.0101583003997803, -0.5095751285552979, -0.5446470379829407, 0.40523549914360046, 0.8998268842697144, -0.74045729637146, 0.25342845916748047, -0.5109695196151733, 2.060533046722412, -0.7682549357414246, 0.6637976169586182, 0.9007510542869568, 2.604870557785034, 0.761573076248169, 1.06014084815979, 1.0867431163787842, -0.7750437259674072, 0.07804683595895767, -0.537860631942749, -0.17270265519618988, 0.9013010859489441, 0.4890749454498291, 0.5231901407241821, -1.3438751697540283, 1.303584098815918, -0.6127921938896179, 2.1462721824645996, 1.2692350149154663, 0.01229848898947239, 0.7903107404708862, -0.32438424229621887, -0.609085202217102, 1.2153422832489014, -0.6538936495780945, -1.022288203239441, -0.8090786933898926, 0.48924872279167175, 0.7464872598648071, -0.20521846413612366, -0.11796420812606812, 0.5455422401428223, -0.5604249835014343, -0.4085067808628082], [-0.21878841519355774, -0.3828442692756653, -0.4868897497653961, 0.804145336151123, -1.1228739023208618, -0.9202104806900024, -0.19040009379386902, -0.4565019905567169, 0.6051456332206726, -0.7847360372543335, -0.3430555462837219, 0.7061281800270081, -0.18941301107406616, -1.3172109127044678, 0.2708909213542938, 0.331248015165329, 0.34523582458496094, -0.9687679409980774, 1.7430628538131714, -0.043345123529434204, 0.2548787295818329, -1.5796736478805542, 1.280702829360962, -0.7906898260116577, -1.5634384155273438, -0.8745813369750977, 0.4682849049568176, 0.371676504611969, 2.2591638565063477, 0.8896644711494446, -1.0362224578857422, -0.91791832447052, -0.06691332161426544, -0.6253023743629456, 1.07872474193573, 0.6535450220108032, -1.8385400772094727, 0.2235630452632904, 0.5952014327049255, -1.9236297607421875, 0.36123764514923096, -0.02574395388364792, -2.042884111404419, -1.5343904495239258, 0.9532259702682495, -0.5772157311439514, 0.2114192396402359, 0.33376583456993103, 0.7016248106956482, -1.8639827966690063, -0.2105148434638977, 0.2852347791194916, 0.4319603145122528, 0.6096197962760925, 0.006719936151057482, -1.0651133060455322, -0.25130850076675415, 0.8766176700592041, 0.17595741152763367, -0.788661777973175, -0.11574675887823105, -0.8214950561523438, -1.258283019065857, 1.0459965467453003, 1.2206645011901855, -1.47047758102417, 0.5162342190742493, -1.015897274017334, 0.0933917686343193, -0.043701380491256714, 0.104016974568367, -0.8750526309013367, 0.6708689332008362, 0.5117177963256836, -0.13296246528625488, 0.43794921040534973, 0.1858402043581009, -0.9368289113044739, -0.9895351529121399, 1.1023606061935425, 2.370936632156372, -0.16269007325172424, 0.9184445738792419, -0.3578227758407593, 0.14715352654457092, 0.8687494993209839, -1.6379716396331787, 0.20370163023471832, 0.4669368863105774, 0.4386776089668274, 0.1553996205329895, 0.05183269456028938, 0.9728264212608337, -0.07646908611059189, 1.5342570543289185, -2.599581241607666, -0.35511842370033264, -0.7142972350120544, -0.4255051612854004, 1.0844067335128784, 0.41254985332489014, 0.7646731734275818, 0.8009647130966187, -1.530007004737854, 1.6708486080169678, -0.9948244690895081, 0.39869388937950134, -0.6016296148300171, -0.4586465656757355, -0.40013235807418823, -0.41129404306411743, -0.2905891239643097, 0.4755799472332001, -0.21405978500843048, -0.8372291326522827, 0.025508938357234, -1.2870147228240967, -1.667189359664917, -0.2867402136325836, 1.593300461769104, -0.8677219152450562, 0.12272022664546967, -0.7167354226112366, 0.1283150464296341, -0.36010122299194336, -1.0494167804718018, 1.0161728858947754, -0.49977537989616394, 1.2518106698989868, 1.23040771484375, -0.22143134474754333, 0.9722068905830383, 0.9736130237579346, 1.4180872440338135, 0.13666097819805145, 1.229709506034851, -1.9525126218795776, -0.04441404715180397, 0.28110939264297485, 0.30452781915664673, -0.5697548389434814, -0.006078365258872509, 1.999306559562683, -0.051396649330854416, 1.823941707611084, 0.5155617594718933, -0.1510877162218094, -0.5581274032592773, 0.31168729066848755, 0.3100261390209198, -0.42115461826324463, 0.2732248902320862, 0.9522078037261963, -0.6913635730743408, -0.062338925898075104, 0.1332840770483017, -0.5709508061408997, -1.9852145910263062, 0.7892765402793884, -1.9397927522659302, 0.8587180376052856, 1.0170776844024658, -0.7216956615447998, -0.8269672393798828, -0.6649279594421387, -0.5586338043212891, 1.4069511890411377, 0.6587293148040771, 0.8081322908401489, 1.031675100326538, 1.0428961515426636, -0.2058410495519638, 0.7492451667785645, 0.224488765001297, -0.9247547388076782, 1.706221342086792, -1.5624282360076904, -1.1502532958984375, 0.8500347137451172, 0.5781911015510559, -0.292122483253479, -1.104427695274353, 0.529359757900238, 1.8191527128219604, 0.14341412484645844, -0.6532330513000488, 0.45409998297691345, 2.3607163429260254, 1.8733773231506348, -1.0022964477539062, -0.8043839335441589, 0.036540012806653976, 0.3129466474056244, -1.330936074256897, -0.7227927446365356, -0.7995572686195374, -0.901566207408905, -1.0668420791625977, 1.8784072399139404, -1.980222225189209, -0.45267897844314575, 1.4743577241897583, -1.5942840576171875, 1.4445680379867554, -0.018759457394480705, -0.08515861630439758, -2.84708309173584, 1.4586131572723389, 1.265751838684082, -0.4027777910232544, -0.5512939095497131, 0.7617220878601074, 2.0672614574432373, -0.6172364950180054, 0.14376242458820343, 1.0222558975219727, -0.5159915089607239, -2.6462197303771973, -0.6537860035896301, 0.10133223980665207, -0.2380276471376419, -1.8480212688446045, -1.8382402658462524, -0.7487730383872986, -0.031287048012018204, 0.4684976637363434, -0.27889350056648254, 2.4865732192993164, 1.4859951734542847, -0.9839745759963989, 0.1699921190738678, 1.8126239776611328, 1.0940585136413574, 0.06454067677259445, 1.4517661333084106, 0.08431912958621979, 0.25132861733436584, 0.42562225461006165, -0.5487107634544373, -0.20099832117557526, 1.4181158542633057, -0.7390625476837158, -0.20635679364204407, -0.11077830940485, -1.1597667932510376, 0.24308054149150848, -1.091416358947754, 0.3359401822090149, 0.3935195207595825, 0.21383146941661835, -0.9029401540756226, 0.5214132070541382, 1.378902554512024, -0.5826486945152283, -0.09961717575788498, 1.7676470279693604, 0.3726934492588043, -1.1395264863967896, -0.917108952999115, -0.0797390267252922, 0.9248839616775513, -0.6945558190345764, -0.5810971260070801, 0.10926319658756256, 0.7070193886756897, 0.8792595267295837, 0.280950129032135, -0.4493080973625183, -1.8078831434249878, 0.9680045247077942, 1.1507899761199951, 1.3833791017532349, 1.5000325441360474, -1.7887887954711914, 0.18457253277301788, -0.5251542329788208, 2.0045392513275146, 0.55949866771698, 1.4080885648727417, -0.7582666277885437, -0.46146947145462036, 0.6503260731697083, 0.7165223956108093, -2.253056287765503, 0.5080817937850952, -0.922771692276001, -0.30408158898353577, -0.9301646947860718, -0.01765500381588936, 0.4678824245929718, 1.0579636096954346, -0.12372717261314392, 0.5331112742424011, 0.4579164981842041, 0.17712092399597168, -0.429715096950531, -1.014656901359558, -0.5676684975624084, -0.6478221416473389, 1.1590832471847534], [0.09540839493274689, 1.2884297370910645, 0.05473881959915161, 1.7574552297592163, -0.7902637720108032, -0.3582096993923187, -0.2451162338256836, 1.2464189529418945, 1.9557634592056274, -1.0338621139526367, 1.1202105283737183, 1.4383413791656494, 0.0925084799528122, 0.7800443172454834, 0.8470777273178101, 0.9863445162773132, 0.5500232577323914, -0.3879639506340027, 0.5183163285255432, 0.8165909051895142, 0.8981987237930298, -1.3664345741271973, -0.21852342784404755, 0.37390556931495667, 1.723891019821167, 0.4223078489303589, 0.208359494805336, -0.45434820652008057, 0.5540561676025391, 0.20888255536556244, 1.5863704681396484, 0.04757639393210411, -1.4710310697555542, -0.9830034971237183, 0.8321278095245361, -0.1742108017206192, 0.8028309345245361, 2.4228570461273193, -1.3750687837600708, -0.9039661884307861, 0.210757777094841, 0.5050517916679382, 0.0620882511138916, 0.6206916570663452, 0.6707520484924316, 1.2493425607681274, 0.22811667621135712, -0.9571383595466614, 0.09525326639413834, 0.4277109205722809, -1.346906304359436, -0.716472864151001, -1.8913474082946777, -0.5901991128921509, 0.3281262516975403, -0.6085930466651917, 0.7072995901107788, 0.5849982500076294, -1.405699610710144, -0.5940032601356506, -0.7717848420143127, -0.5412542819976807, 2.175389051437378, 0.6405656337738037, 0.26165464520454407, 0.7654382586479187, 1.0340713262557983, -2.4266164302825928, -1.9289988279342651, 0.8184314966201782, -0.1564178615808487, 0.3285163342952728, 0.6369021534919739, -0.019910458475351334, 1.0585169792175293, -0.8070477247238159, 0.7105692028999329, 1.4020435810089111, 0.07872063666582108, -1.1140742301940918, 2.5106329917907715, -0.6565269231796265, 1.612818717956543, -0.7035643458366394, 0.42214471101760864, 1.3028333187103271, -0.2911853492259979, -0.9544154405593872, 0.0019600584637373686, -0.7073408961296082, 1.3310770988464355, -1.0632442235946655, 0.5979937314987183, -0.9371483325958252, -0.29908570647239685, -0.8078247308731079, -0.7270150780677795, 0.4501534104347229, 0.23814286291599274, -0.9160001277923584, -1.1000840663909912, -0.9040883183479309, 2.0432510375976562, 0.3415895700454712, -1.0174273252487183, 0.6923751831054688, -0.7340743541717529, -0.3502095937728882, -0.8767633438110352, 0.32712605595588684, -0.2531251311302185, 1.044091820716858, 1.2918795347213745, 0.14469806849956512, -0.5783113241195679, -1.6466362476348877, 0.9772090911865234, 0.07928330451250076, 1.2931371927261353, 0.8562818169593811, -1.2473901510238647, 0.5331061482429504, -0.26451289653778076, -0.09945759922266006, -1.2342252731323242, -0.12941449880599976, -1.1259124279022217, 0.44937512278556824, 1.9311120510101318, 0.27567169070243835, -0.24421310424804688, 0.6423103213310242, 0.7461054921150208, 0.051870908588171005, 0.41262057423591614, 0.8285340070724487, 0.3919049799442291, -2.0940330028533936, 1.4418550729751587, -0.2333010584115982, -0.06143759936094284, 1.1441811323165894, -1.0071779489517212, 1.5885032415390015, -2.059314250946045, 1.306086540222168, 0.1448543667793274, -0.2033003568649292, 1.2131197452545166, 0.5792732834815979, 0.3014272451400757, 0.8841063976287842, 0.425269216299057, -0.26302772760391235, 0.3523087799549103, -0.3288622200489044, 0.7153083086013794, 1.0583813190460205, -0.9990153908729553, 2.056579828262329, -2.3445968627929688, -1.3270360231399536, -1.2706224918365479, -0.7885705232620239, -0.5204498767852783, 1.074008822441101, -0.2497689425945282, 0.15751348435878754, -0.07291030883789062, 0.7862139344215393, -0.2850542664527893, 0.24010063707828522, -1.5775139331817627, -0.2416401207447052, -0.9590246677398682, -0.6890437006950378, -1.183153748512268, 1.0396392345428467, -0.5800922513008118, -0.5109879374504089, 0.7019582986831665, -0.9552087783813477, -1.0451897382736206, 2.199887990951538, 0.3375859260559082, 1.7583718299865723, 1.6460087299346924, 0.4183703064918518, 0.226227805018425, 0.1455678790807724, -0.7169686555862427, -1.1820087432861328, -0.42063847184181213, 0.4730433523654938, -0.6844381093978882, 1.0513453483581543, 1.598765254020691, 1.1113231182098389, 0.8248271346092224, 0.6987872123718262, 0.9974408745765686, -0.3343857228755951, 0.51120525598526, 0.7072862386703491, -0.3526402711868286, -0.7449772953987122, -0.25204625725746155, 1.4622623920440674, 1.9463621377944946, 0.36616453528404236, -0.989448070526123, 0.34510472416877747, -0.3581189513206482, -0.594940721988678, 0.6005410552024841, 0.3731633424758911, 0.8433374166488647, 0.379398912191391, 0.2833322286605835, 0.3823336064815521, 0.8213902711868286, -0.23561403155326843, 1.4577867984771729, -0.516182541847229, 1.5494930744171143, -0.48860788345336914, 1.5855274200439453, -1.7772971391677856, 0.20154789090156555, -0.9696130156517029, -0.9193544983863831, -0.7898948192596436, -1.548122525215149, -0.9738646745681763, 1.0054059028625488, -1.0198203325271606, -1.8171197175979614, -0.7210602760314941, -0.029089907184243202, -0.19392120838165283, -0.7923528552055359, -0.6215075850486755, 1.7967476844787598, 1.3842443227767944, -1.3171666860580444, 1.1735503673553467, 0.6424244046211243, 0.5456864833831787, -1.350035309791565, -0.11612197756767273, -0.9739155769348145, -1.6183093786239624, 0.6919934153556824, -1.2778877019882202, -0.5877151489257812, -0.09087377041578293, 0.7751235365867615, -0.1202276423573494, 0.47099217772483826, -1.316542148590088, -0.7219952940940857, 0.8653804659843445, 0.10689201951026917, -0.960200309753418, 0.5530439019203186, 1.641067624092102, 1.274282455444336, -0.16441723704338074, 0.789910078048706, 1.58686101436615, -0.603471577167511, 0.2759944200515747, -0.881112813949585, -0.18621192872524261, 0.5442151427268982, -0.8406463861465454, 0.5218148827552795, -0.8004204630851746, -0.655402421951294, 0.25106480717658997, -0.06439980864524841, 0.3362986445426941, -0.24562768638134003, -0.3273693919181824, 1.2646735906600952, 1.8269280195236206, 1.3573507070541382, -0.048936229199171066, -0.1749519407749176, 0.8983026742935181, -1.0297131538391113, -0.18586836755275726, -0.10905468463897705, 0.3589601218700409, -0.2748260200023651, -0.26461875438690186, -0.40653324127197266, -0.8840351104736328, 1.606040358543396, 0.18203066289424896], [1.396270513534546, -1.5595169067382812, 0.09332334250211716, -0.967516303062439, 0.20386768877506256, 0.7356457114219666, -0.8641281723976135, -0.18765480816364288, 0.6785885095596313, 0.30566176772117615, 1.1918487548828125, 0.6321825981140137, -1.1794127225875854, -1.3814555406570435, 1.357527494430542, 0.5267648696899414, 0.4110993444919586, 0.0019283501897007227, -1.3139560222625732, 0.8656133413314819, -0.7867791056632996, 0.7755028009414673, -0.6176953315734863, -0.9644840359687805, 1.1519807577133179, -0.3459576964378357, -0.5132832527160645, 1.6687551736831665, -0.3025623857975006, 1.7132341861724854, -0.14269186556339264, -1.1151728630065918, 1.481776475906372, -0.12844550609588623, -1.0528075695037842, 0.8210747838020325, -0.39186322689056396, 2.1830241680145264, -0.15888692438602448, -0.9882505536079407, -0.35717082023620605, -0.38037580251693726, -0.6909468173980713, 1.0666134357452393, -1.6991249322891235, -0.9017069935798645, -0.5829806327819824, -1.0253609418869019, -0.5571646690368652, 1.1559616327285767, 0.4861346483230591, 2.2399988174438477, -0.37924471497535706, 0.27697885036468506, 1.817725658416748, 0.6550853848457336, 1.135662317276001, -0.9108474254608154, 0.4850606322288513, 0.13711689412593842, 0.017984170466661453, -0.7038444876670837, -0.31793317198753357, -0.5142248868942261, 1.1600453853607178, -1.386353850364685, -0.7911132574081421, 1.355850338935852, 1.3228644132614136, -0.4665941298007965, 0.25571995973587036, 1.6420892477035522, -0.6332849860191345, -0.9840949177742004, 0.568981945514679, -1.4236071109771729, 1.1804226636886597, -0.4246617555618286, 0.37701645493507385, -0.6930463910102844, 0.5772082209587097, -1.0170423984527588, -0.2987166941165924, -0.7972785830497742, -0.5007688403129578, 1.1441996097564697, 1.3344351053237915, 1.8611737489700317, 0.09198059141635895, -0.8260261416435242, 0.4688943028450012, -1.594132661819458, -0.97541743516922, 0.687426745891571, -0.2971048355102539, 0.7808051109313965, 0.4112437665462494, -0.16344952583312988, 0.4311244487762451, -0.0021733713801950216, -0.9315745234489441, 1.894287347793579, 1.2402660846710205, 0.6725277304649353, -0.9536502361297607, 0.4641821086406708, -0.9005966782569885, 0.18841764330863953, 1.1696208715438843, -0.37562838196754456, 0.28886595368385315, -0.8208397030830383, 0.6991629004478455, 1.3122005462646484, 0.16956467926502228, 1.8505758047103882, 0.1496097892522812, -1.8462555408477783, 1.1429966688156128, 0.3914388120174408, -0.8603700995445251, -0.2427964210510254, 0.04291922599077225, 0.9235193729400635, 0.6400193572044373, -1.6289894580841064, 0.7407673597335815, 1.0303152799606323, -1.1215288639068604, 1.228930950164795, -0.24046078324317932, -2.953901529312134, 0.8789953589439392, -0.005253884941339493, 0.9472703337669373, -1.3802177906036377, -0.43576595187187195, 1.147720456123352, -0.6340668201446533, -0.9779866337776184, 2.0548388957977295, -1.039405107498169, -1.3149853944778442, -0.04803670942783356, -0.33742696046829224, -0.8431496620178223, -0.8398137092590332, 1.2033616304397583, -0.38352587819099426, -0.48231855034828186, -1.045480489730835, 0.48584482073783875, 0.10702618211507797, -1.741439938545227, -0.06763201206922531, -0.3723360002040863, 0.20054978132247925, -0.6008815765380859, 1.4647024869918823, 0.6005651950836182, 0.45237547159194946, 2.159101963043213, 0.8435078263282776, -0.24393990635871887, -0.5032572150230408, 0.7441071271896362, 1.2243928909301758, -0.6461738348007202, -0.6798309087753296, 1.4068717956542969, -0.26536989212036133, -1.3602123260498047, 0.3568335473537445, 0.0963037759065628, 0.6174541711807251, -0.4121571481227875, -0.5104682445526123, -0.2729097306728363, -0.44252872467041016, -0.6127257347106934, -0.0204295851290226, -0.7666821479797363, 0.6642618179321289, 0.39538347721099854, -0.20426355302333832, 1.7996479272842407, 1.3193222284317017, 0.6107149720191956, -0.6205081343650818, -1.1968350410461426, 0.7102684378623962, 0.6661563515663147, -0.7778962254524231, 0.9852937459945679, -0.9299761056900024, 0.7425593137741089, 0.9995227456092834, -0.15088270604610443, -0.4989676773548126, -1.133697509765625, 1.4065388441085815, 0.507884681224823, 1.1975055932998657, 1.5060124397277832, 1.7797794342041016, -0.27205410599708557, -0.07709133625030518, -0.8307898044586182, 1.7762597799301147, -0.5229736566543579, -0.484220027923584, -0.6900386214256287, 0.3469838798046112, 0.12583334743976593, -0.5719417333602905, -0.6863059401512146, -1.148013949394226, -0.38693302869796753, 0.31916049122810364, -1.0560818910598755, -0.2535877525806427, -1.3795684576034546, -1.6966954469680786, -1.5160675048828125, -0.532248318195343, -1.2352076768875122, -0.26555323600769043, -0.2504599094390869, 1.3405628204345703, 0.6513122916221619, -0.30576035380363464, 1.326195240020752, -0.6926315426826477, 0.4069686233997345, 0.13183414936065674, 0.47698459029197693, -0.49282994866371155, -1.16318678855896, 0.07839689403772354, 0.7251613736152649, -0.37510842084884644, -0.4153445363044739, -2.5428993701934814, -0.7810837030410767, 1.7685598134994507, 0.18661057949066162, 1.519126296043396, 1.932180643081665, -1.7837674617767334, 2.233119010925293, 1.7304315567016602, -0.1343483030796051, -0.16799791157245636, 0.42882952094078064, 0.3216450810432434, -0.39586374163627625, -0.4321228861808777, 1.335514783859253, -0.5329327583312988, -1.4382374286651611, -1.0871391296386719, -0.4731402099132538, -0.08828707784414291, 0.5467426776885986, -0.3600767254829407, 0.019089264795184135, 0.9309607744216919, 1.8634109497070312, 1.3796563148498535, -0.6708492040634155, -0.3139927089214325, -0.9415470957756042, -0.39336246252059937, 1.249511480331421, 0.67637038230896, 1.3145523071289062, 0.3753865957260132, -0.066436268389225, 0.4823259711265564, 0.0019426322542130947, 2.6083943843841553, -0.7404011487960815, -1.0712441205978394, 0.3210986852645874, 0.35470816493034363, 0.038146790117025375, 2.0150015354156494, -1.0857137441635132, -0.45961034297943115, -0.28038454055786133, -0.916327178478241, -2.5353219509124756, 2.57393741607666, -1.2183011770248413, 0.30961844325065613, -0.3921079933643341, -1.5144901275634766, 1.1972739696502686, -0.6355929374694824, -1.3317008018493652], [-0.7834500074386597, 1.1010370254516602, -1.6795274019241333, -0.04328729584813118, -1.189836025238037, -1.5883020162582397, 0.05984533950686455, -0.25724273920059204, 0.47675621509552, 0.13550633192062378, -0.8618282079696655, 0.9714530110359192, 0.10518922656774521, 0.4324290156364441, -0.1427583247423172, 0.48177653551101685, 1.1429097652435303, 0.9843730926513672, -1.5398542881011963, 1.0137286186218262, -0.1266944706439972, -0.9729692935943604, -0.7497849464416504, 1.0648196935653687, 0.16524425148963928, -0.4232823848724365, 0.13772788643836975, 2.4047179222106934, -0.5742807388305664, -0.5360061526298523, -0.4594819247722626, 2.3633129596710205, -1.3733513355255127, 0.8471317887306213, -0.15937753021717072, 0.7522261142730713, -1.1921907663345337, -0.7406664490699768, 0.2946969270706177, -0.7179281115531921, 0.9243808388710022, -0.1157928854227066, 1.9870874881744385, 0.2743341028690338, -1.0712823867797852, 0.07840975373983383, -1.1899242401123047, 1.6291059255599976, -0.5832770466804504, -0.34530067443847656, 0.10658848285675049, -1.8181101083755493, 0.49359163641929626, 0.038894638419151306, 0.5457159876823425, 0.8154865503311157, 0.5221807956695557, 2.0039784908294678, -0.6484250426292419, 0.027347829192876816, 0.5390104055404663, -0.41222599148750305, -0.23382899165153503, -0.160634383559227, 0.9355896711349487, 0.369040846824646, 1.3023490905761719, -0.7124534249305725, -1.2727552652359009, -1.8622324466705322, 0.041302673518657684, -0.22328411042690277, 2.472379446029663, -1.317125678062439, 0.7885501384735107, 0.008018489927053452, -1.8079437017440796, -0.8813907504081726, -1.7385904788970947, 0.09328021109104156, 1.809897780418396, -0.1078500971198082, 1.7168740034103394, -1.0780056715011597, 1.088187336921692, 0.2344193160533905, -1.1917264461517334, -1.6102914810180664, -1.0950387716293335, -1.3120821714401245, 1.045823574066162, 0.5359148979187012, -0.38908499479293823, -1.1201093196868896, -1.8319129943847656, 0.05477287992835045, 0.35893476009368896, -0.33327582478523254, -0.4814698398113251, -0.9816526174545288, -0.23028816282749176, 0.8912185430526733, 1.0509203672409058, 0.15697315335273743, -0.7052459120750427, -0.9000492095947266, 1.2738977670669556, 0.7316312193870544, 0.5108537077903748, -1.2367268800735474, 0.6955621242523193, -0.4469398856163025, 2.3129544258117676, -1.1079792976379395, 1.1757593154907227, 1.4280537366867065, -0.692794144153595, -1.0596026182174683, -0.09863992035388947, -0.13439323008060455, 1.1157894134521484, -1.1833012104034424, 1.427157998085022, -0.5906500816345215, 0.7291455268859863, -0.2855282425880432, -0.2107965052127838, 1.7645788192749023, -1.248451828956604, 0.5816020369529724, 0.23960381746292114, -0.48896917700767517, 0.9069987535476685, -0.6714316010475159, -0.5755988955497742, 0.0791706070303917, 1.2350428104400635, 0.06831944733858109, -2.036378860473633, -1.9579561948776245, 0.31868651509284973, 0.6320416927337646, -1.3073608875274658, 0.762893557548523, 2.1928458213806152, 0.10766682773828506, -0.8501541614532471, -0.3855811655521393, 1.0427310466766357, -0.7663475871086121, -0.16371296346187592, 0.06603632122278214, -0.7881740927696228, 0.864158570766449, -1.5305685997009277, -2.2659151554107666, -3.641219139099121, 2.2740848064422607, -0.6847797632217407, -0.09783052653074265, -1.5586775541305542, -0.07929202914237976, -0.12827129662036896, -1.164993405342102, -0.022950729355216026, 0.7032674551010132, 0.41090357303619385, 0.006644231267273426, 0.7044907808303833, -0.6852102875709534, -0.5814005136489868, 0.22974854707717896, -0.29551562666893005, -0.2346627414226532, 1.8210246562957764, 0.2535878121852875, -1.6577773094177246, 1.870209813117981, -1.561216950416565, -2.049220561981201, 0.13506603240966797, 2.7016100883483887, -0.3130980134010315, -0.6772130131721497, 0.6349092721939087, 1.1948579549789429, -0.47012126445770264, 1.5720252990722656, 0.41719844937324524, 0.21960169076919556, 1.4531638622283936, 0.4606047570705414, -1.3019084930419922, 1.2786580324172974, -0.030212100595235825, -0.7053149938583374, 1.1518641710281372, 0.8875748515129089, 0.7304131388664246, -0.44288185238838196, 0.8336361646652222, 0.740455687046051, -0.6804479360580444, 1.572371006011963, 0.6019089221954346, -0.28324106335639954, -0.901485800743103, 1.1282559633255005, 0.3587910532951355, 0.24886484444141388, 0.7893130779266357, 1.0447973012924194, -0.46710649132728577, 1.6464152336120605, -1.6829502582550049, 0.46570736169815063, 0.5122026801109314, 0.24092014133930206, 0.5921880006790161, 0.8327459692955017, -0.09110676497220993, -1.3165470361709595, 1.374643325805664, 1.0190094709396362, 0.22338488698005676, -0.7483569383621216, -1.8458788394927979, -0.003896932350471616, -0.03329841420054436, 0.07781224697828293, -1.8623871803283691, -0.5019312500953674, -0.033955641090869904, 0.5017656087875366, -0.2373729795217514, -0.669576108455658, 0.5414879322052002, -0.8616077303886414, 0.4173712730407715, -1.5205395221710205, 0.23683999478816986, -1.2603756189346313, -0.16373738646507263, 1.5774099826812744, -0.12012270838022232, 0.3296591341495514, -1.482089638710022, 1.2739368677139282, 0.5027855634689331, 0.6039206981658936, -0.6563060879707336, -0.8005121946334839, 1.4426954984664917, 0.6779412627220154, 2.265935182571411, 2.27807879447937, 0.1648799329996109, -0.9290609359741211, -0.7826389670372009, 0.38203176856040955, 0.1984139233827591, -1.7085250616073608, -1.251900315284729, 0.5557539463043213, -0.10576802492141724, 1.6751093864440918, 1.3969627618789673, 0.05231015011668205, -0.41777560114860535, -0.42879822850227356, -0.627826988697052, -2.120659589767456, 0.47056707739830017, 0.09370840340852737, -0.7135334014892578, -0.8577684760093689, -0.402163028717041, -0.20521460473537445, 0.05282910540699959, -0.8677829504013062, 0.03271366283297539, -1.0871366262435913, 0.32266175746917725, -0.17169639468193054, -0.003908879589289427, -1.9558733701705933, -0.8353013396263123, -0.34073132276535034, 0.42818328738212585, -1.3657516241073608, 1.929883360862732, 0.589676022529602, -0.06728231906890869, -0.23487962782382965, -1.5977579355239868, -0.9485148787498474, 1.3117399215698242, -0.38569167256355286, -0.7747630476951599, -0.7519703507423401], [0.2944200932979584, -0.8996548652648926, 0.15289495885372162, 0.01593816839158535, -1.8215065002441406, -0.8507654666900635, 1.7722498178482056, -0.27213987708091736, -1.5505727529525757, 0.7218741774559021, -0.5346392393112183, 2.435983419418335, -1.554115653038025, -0.03771083801984787, 0.8403347730636597, -1.757516860961914, 1.5398952960968018, -1.0035679340362549, -0.23928816616535187, 1.8003586530685425, -0.25447025895118713, -0.2120588719844818, 1.5606284141540527, -0.5357367396354675, 1.3317182064056396, -0.9697251319885254, -0.4426538348197937, -0.1431090533733368, -0.24074120819568634, 0.7841503024101257, 0.40245845913887024, 1.4770179986953735, 1.389910101890564, 0.06723439693450928, 1.5384258031845093, 0.6451107263565063, -1.1486680507659912, 0.18130844831466675, 1.0909135341644287, 0.5068342089653015, 0.5571607947349548, -0.8919104337692261, 0.1383926272392273, -0.3730901777744293, 0.13493436574935913, 0.27974244952201843, -0.5050002932548523, 0.05463592708110809, -0.14486373960971832, -0.7536296248435974, 0.3142930269241333, 1.2760459184646606, -1.2922240495681763, -0.37299254536628723, 0.16528117656707764, -1.591194987297058, 1.2223029136657715, 0.10674799233675003, -0.6137231588363647, 0.39578548073768616, 0.09416057914495468, -0.038141217082738876, 1.065657138824463, -0.34536096453666687, 0.33731958270072937, 2.3709216117858887, 0.5054970979690552, 0.10985510796308517, -0.3839874267578125, 1.923085331916809, 0.8439601063728333, 0.363176554441452, 0.2600778639316559, -1.6083124876022339, -0.025621447712183, -2.419480085372925, 2.7402420043945312, -0.6138614416122437, 0.7661066651344299, -0.14403338730335236, -1.7781661748886108, -0.4485487639904022, 0.32796624302864075, 0.7444806098937988, 0.5514978170394897, -0.8713759779930115, -0.016854412853717804, -0.08024149388074875, -0.06304982304573059, -0.5912185907363892, 0.30916503071784973, -0.9265955686569214, 0.09839308261871338, 1.2762054204940796, 0.7454366087913513, 0.6545495986938477, 0.6716704368591309, -0.07326885312795639, 1.2570174932479858, 1.0900545120239258, -0.5123361945152283, 2.017427682876587, -0.16453519463539124, 0.9748787879943848, 0.7277100086212158, 0.6818008422851562, 0.29869160056114197, -0.3528933823108673, -1.9259936809539795, 0.43500176072120667, 0.1839965134859085, 0.5060321688652039, 1.0029038190841675, 0.5157371163368225, 0.5090888738632202, 0.9973140954971313, -0.7427423000335693, -0.18066954612731934, -0.364338219165802, 0.010506768710911274, -0.7753818035125732, 0.836410641670227, 0.9392369389533997, 0.3112449049949646, 1.4289263486862183, 1.3008475303649902, 1.2253379821777344, 0.5181260704994202, 0.51351398229599, 0.09883630275726318, 1.2054086923599243, -0.10708608478307724, -0.15069960057735443, 0.6388052701950073, -0.22009095549583435, -0.6847406625747681, -0.8622314929962158, -0.45619437098503113, -0.43778812885284424, 1.0601707696914673, -0.28999873995780945, 1.6549887657165527, 0.7672297954559326, -0.3091261386871338, -0.7175946831703186, 0.011326978914439678, -0.7136082649230957, 1.1715292930603027, -1.4118096828460693, 0.03535875305533409, -1.3343174457550049, -1.354174017906189, -0.7653031945228577, 1.6513433456420898, -0.23258008062839508, 0.5019896030426025, -0.09423672407865524, 1.155178189277649, 1.3263767957687378, -0.5297865271568298, 0.3361874222755432, -0.06731268763542175, 0.7916456460952759, 0.23958246409893036, 0.30031079053878784, 1.3361270427703857, -0.33081042766571045, -0.5472415089607239, 0.11750861257314682, -0.24723149836063385, 0.7093012928962708, 0.8223453164100647, 0.12170740962028503, 0.6584001183509827, -1.2953981161117554, 0.6083007454872131, 0.8963477611541748, 0.8188859820365906, 1.972934365272522, 0.7092875838279724, 0.3757138252258301, 0.34601476788520813, -1.0525826215744019, -0.9345829486846924, 0.44042375683784485, -0.3754512071609497, 0.0455130971968174, -1.077606439590454, 0.20457303524017334, 2.069835901260376, -0.14919570088386536, -0.13164710998535156, 1.38285493850708, 1.161177158355713, -0.362613320350647, 1.0203050374984741, -0.6664542555809021, -0.27522969245910645, -0.31841006875038147, -1.1371515989303589, 1.9544672966003418, -0.9191479086875916, -1.0414931774139404, -0.5541605353355408, -0.9926573038101196, 0.5313681960105896, -1.5212849378585815, 0.4168262481689453, 0.24007658660411835, -2.0199074745178223, -0.592368483543396, 1.2729315757751465, -0.41327834129333496, -0.13978064060211182, -0.45379236340522766, -0.19749104976654053, -0.29452288150787354, 0.35086676478385925, -1.0070966482162476, -0.7664567828178406, -0.5186242461204529, 0.036708686500787735, -0.2073737382888794, 1.608339786529541, -0.6114705204963684, -1.50114905834198, -0.524599015712738, 1.040711522102356, 1.1771488189697266, 0.00937072467058897, 0.657268762588501, -0.00509418360888958, 1.843161940574646, -1.8821418285369873, 0.46820369362831116, -0.539537250995636, -0.025058867409825325, 0.447369247674942, 0.3534012734889984, -0.08057765662670135, 0.8845507502555847, -1.8872865438461304, -2.2753636837005615, -0.8825585246086121, -1.6797716617584229, 1.2732493877410889, 0.7500262260437012, -2.0654749870300293, -0.5857565999031067, -0.18964777886867523, 2.025803565979004, 1.0066626071929932, 1.314231038093567, 0.8593308925628662, -0.1508394181728363, -0.7026797533035278, 1.8626643419265747, -2.168153762817383, 0.7048511505126953, 0.3812108635902405, -0.03477747365832329, -1.1487611532211304, -0.24421092867851257, -0.904038667678833, 3.0433506965637207, 1.5484540462493896, 1.8236682415008545, -0.3662409782409668, -1.0903645753860474, 0.7651893496513367, -0.5808612704277039, 0.47909969091415405, 0.9110360741615295, 1.7636873722076416, -0.23255303502082825, -0.7608057856559753, -0.02459494024515152, 1.0558286905288696, 0.1459515392780304, -1.1360554695129395, 0.7875022888183594, 2.402225971221924, 0.05437399446964264, -0.1720348447561264, 1.666881799697876, -0.710141122341156, -1.4271875619888306, -1.447773814201355, -0.5360536575317383, 1.5270545482635498, 0.10642658174037933, -0.09317656606435776, 1.5891780853271484, 0.6722626090049744, -0.7829464077949524, -0.6120051741600037, -0.7853710055351257, -0.5982580184936523, 0.6588560938835144, -1.8498482704162598], [-0.07449476420879364, -0.32175183296203613, 0.7774246335029602, -0.6851149797439575, 2.4141337871551514, 0.70858234167099, 0.577165961265564, 0.40094664692878723, 0.19569778442382812, 2.4166102409362793, -1.114823818206787, 0.3228856027126312, 1.5686588287353516, 0.6028648018836975, -1.193993330001831, 0.5296295881271362, 1.54062819480896, -0.1326095014810562, -1.5155695676803589, -0.9320169687271118, -0.16911986470222473, 0.006916873622685671, 0.6630700826644897, -0.7495511174201965, 0.04111197218298912, 1.3470277786254883, -0.9666072726249695, 0.6892184615135193, -1.185621976852417, -0.08616428822278976, 0.3813445568084717, -0.2721722424030304, 0.7574285268783569, 0.4710269272327423, -1.5356289148330688, -1.481685757637024, 0.4868103563785553, 0.897699773311615, 0.045610398054122925, 0.36630138754844666, 1.5208168029785156, -0.6129677891731262, 0.11692728847265244, 0.39770346879959106, -0.4488104581832886, 0.05383070185780525, 0.6516789197921753, -0.5662769079208374, -0.041903551667928696, -0.6166262030601501, 1.741092562675476, 0.23996570706367493, 1.2509255409240723, 1.8593780994415283, 0.5412168502807617, 0.35639849305152893, -2.3206372261047363, 1.0688490867614746, -1.160198450088501, 0.18786633014678955, -0.11176394671201706, 0.6926396489143372, 0.7278075814247131, 0.007691855542361736, 0.4259704351425171, 0.4722457230091095, 0.22001662850379944, 1.0521951913833618, 0.0351925864815712, -1.1013928651809692, -0.7046759724617004, -0.06814519315958023, -0.14413249492645264, 1.3357007503509521, -0.4707893133163452, -1.3551009893417358, 2.688572883605957, 1.1265329122543335, 0.3092900216579437, 0.11888289451599121, -0.2958220839500427, 1.3415119647979736, -0.8966103792190552, 0.809721827507019, 0.10897907614707947, 0.8489800095558167, 1.032346487045288, 0.8603007197380066, 0.6258988976478577, 0.7310753464698792, -0.18428106606006622, 0.36643314361572266, 0.4134528636932373, 0.2994164824485779, 0.896379828453064, 1.81575345993042, 0.16462059319019318, -0.1378636211156845, -0.04441000148653984, -1.7080734968185425, 0.9063299894332886, 0.07770241051912308, -1.4242377281188965, -0.6298155784606934, 1.2108571529388428, -0.4814094603061676, 1.0013890266418457, 0.271513968706131, -1.1733958721160889, 0.3720051646232605, -0.3935936987400055, -0.2490077018737793, 0.5423423647880554, 0.08138974756002426, 1.3040809631347656, 2.07149076461792, -1.5535030364990234, 1.5375312566757202, -0.4661743640899658, 0.15295736491680145, -0.823190450668335, 0.10300179570913315, 0.0335957407951355, 0.09193895012140274, 0.1966407746076584, 0.11281301081180573, -1.2869431972503662, 0.8145740628242493, 1.093970537185669, -0.22879686951637268, 0.7739909291267395, -0.542051374912262, 0.04190181568264961, -1.202731728553772, 1.6393635272979736, -1.4393548965454102, -1.4695757627487183, 1.6221789121627808, -0.45017287135124207, -2.7548718452453613, -1.5447007417678833, -2.124565839767456, -0.013090584427118301, -0.32094117999076843, -0.7051334977149963, -0.7064141035079956, 1.6361297369003296, 0.22450320422649384, 0.0701216608285904, -0.3929803967475891, -0.6210219264030457, -1.2619181871414185, 0.214940145611763, 0.5709697008132935, 0.37146425247192383, 0.9670475721359253, -0.8665258288383484, 1.433212161064148, 0.11694575846195221, -0.7093626856803894, -0.5465080142021179, -0.913144588470459, 0.07434843480587006, -0.2190207839012146, -0.3175617456436157, 0.7674611806869507, -0.02680329792201519, -1.1335029602050781, 0.38083919882774353, -1.3491424322128296, 0.09654944390058517, 0.6498052477836609, -0.28845393657684326, 0.8874277472496033, -0.3960666358470917, -1.2648173570632935, -0.99434894323349, -1.0080342292785645, -1.003146767616272, -0.414095014333725, -0.6689449548721313, 0.8114811182022095, 1.2928965091705322, 0.35893937945365906, 2.1479716300964355, -0.24616704881191254, -1.5921627283096313, 0.6847789883613586, -1.3255523443222046, 0.102327361702919, -0.4333608150482178, 0.014392857439815998, 0.1395057737827301, -1.5169399976730347, 0.7911163568496704, 0.2277502417564392, -0.39267483353614807, 0.12454692274332047, -0.4786226451396942, 1.833133578300476, -0.9723349213600159, -0.5425803065299988, -1.7660281658172607, 1.0509096384048462, 1.0789623260498047, -0.15904536843299866, 1.0241804122924805, -0.3195667266845703, -1.3059645891189575, 1.120129942893982, 0.3613458573818207, 0.932457685470581, 0.605388343334198, -0.26585084199905396, 0.1417870670557022, -0.8309495449066162, 2.443605422973633, 0.5177825093269348, -2.0379302501678467, -0.6735527515411377, -0.8001906871795654, 1.3646979331970215, -1.0723049640655518, 0.4525150656700134, 0.972997784614563, -0.45908474922180176, -0.10309392958879471, 0.39030373096466064, -1.1078583002090454, 0.5229632258415222, 0.019108669832348824, -0.0945536270737648, -2.1853208541870117, -0.033486079424619675, -0.6638992428779602, 0.4211224615573883, -0.40087470412254333, 0.24288484454154968, -0.9948673844337463, 0.41827526688575745, -0.8003957271575928, -0.4661554992198944, -0.3050709664821625, 0.8302929997444153, 0.1974438577890396, 0.8128441572189331, 2.616874933242798, -1.0318058729171753, 0.9328562617301941, -0.7956556677818298, -0.5342192649841309, -0.4060245156288147, -1.2165653705596924, -1.4840492010116577, 0.939326822757721, 0.8945760726928711, -1.5432597398757935, 0.6437269449234009, -0.3252364695072174, 0.1937405914068222, 0.46026498079299927, -1.968153715133667, -0.8582332134246826, -0.5406765341758728, -0.7362525463104248, 0.2846885323524475, -2.734666347503662, -1.0410798788070679, -0.2684684097766876, -0.8646034598350525, -1.6801058053970337, -1.4818907976150513, -1.754310965538025, 0.9082897305488586, 0.36467429995536804, 0.204549178481102, -1.3767707347869873, -0.34715327620506287, -0.20677106082439423, 1.3292866945266724, -0.7747525572776794, -0.7126035690307617, 0.9428292512893677, -0.06554152071475983, 0.1641330122947693, -0.7363325953483582, 0.08645004779100418, 1.5706720352172852, 0.6045458912849426, 1.7106833457946777, -0.7281454205513, -0.35532090067863464, -0.3200923502445221, 0.20255570113658905, 1.09477961063385, -0.5011136531829834, -1.7422727346420288, -0.6567286252975464, 1.010805606842041, 0.05339939147233963], [0.9900215268135071, -0.9209141135215759, -1.2567245960235596, -0.14187991619110107, 0.1404329538345337, -0.7556065917015076, -0.8932265043258667, 1.158229112625122, -1.1613643169403076, 1.7536487579345703, -0.08786708861589432, -1.2893762588500977, 0.8377344012260437, 0.8493630886077881, -1.9585474729537964, -1.7994533777236938, 0.3409712612628937, -0.7119781970977783, 1.7809151411056519, -1.0049113035202026, 0.6334220767021179, -1.1001051664352417, 0.11689205467700958, 0.8566160202026367, -0.011427821591496468, -1.214231252670288, -0.9286041259765625, -0.23203562200069427, -2.4831154346466064, 1.1137785911560059, -0.9742306470870972, 0.0026933536864817142, -0.4991312325000763, 0.5878162384033203, 0.19520799815654755, -2.535818338394165, -1.037339448928833, -0.4168802797794342, -0.34230971336364746, -0.8431089520454407, -0.5791581869125366, -0.29425570368766785, -0.6808003187179565, 0.323901891708374, 0.8926408886909485, -0.5798613429069519, -1.1930092573165894, 0.8413692712783813, -0.4901295304298401, 1.2196601629257202, 0.6550028324127197, 1.3325556516647339, 1.0927481651306152, 0.5496366620063782, -0.5119391679763794, 0.10852552205324173, -0.27239128947257996, 1.2161221504211426, -0.4131394922733307, 0.11385940760374069, -1.407979130744934, -0.314031720161438, -0.7438985705375671, 0.42210039496421814, 1.1882965564727783, 1.352266550064087, -1.0997636318206787, -0.09657061845064163, -2.37056040763855, -0.40481096506118774, 2.456076145172119, 0.7024106979370117, -0.20792068541049957, 0.31261882185935974, 0.5874614119529724, 0.5743663311004639, -1.5773797035217285, -1.1496773958206177, 0.30526265501976013, -0.31704679131507874, 0.3279275596141815, -1.0784457921981812, 0.5459470748901367, -0.5355137586593628, -0.18620868027210236, 0.6777984499931335, -0.9789649844169617, -0.39109542965888977, -0.19472108781337738, -0.19631320238113403, -0.3508094251155853, 1.524769902229309, 0.6899957060813904, -1.3627759218215942, -0.8933525085449219, -0.7101370096206665, 0.4175049960613251, -0.9356448650360107, -0.29560574889183044, 0.743823766708374, -1.6482514142990112, -0.22532717883586884, 0.21577095985412598, 1.1027048826217651, -0.3228229284286499, -0.18571622669696808, -0.8361209034919739, 0.37701699137687683, -0.604237973690033, 1.4104399681091309, -0.4899275004863739, -0.9845213890075684, 1.0250310897827148, 0.27018478512763977, 0.5345684289932251, 0.39433592557907104, 0.0501236729323864, 0.21221551299095154, -0.37835246324539185, -0.273345410823822, 1.904109239578247, 0.24976558983325958, 2.3781189918518066, -0.20170162618160248, -1.2750999927520752, -2.012345552444458, -0.15616025030612946, -0.5849813222885132, 0.024920467287302017, -1.2485016584396362, -1.0906956195831299, 0.4589287340641022, -0.6421958804130554, 0.5738759636878967, -1.5602855682373047, -0.25868457555770874, -1.4350167512893677, -0.5704236626625061, -0.12786215543746948, -0.5463816523551941, 0.43220630288124084, 0.30507802963256836, -0.0745508000254631, -0.5376353859901428, -1.0517473220825195, -0.967454731464386, -1.0169391632080078, 1.6180957555770874, -0.18694736063480377, -1.323702096939087, 1.30238676071167, 1.7514981031417847, -1.1065847873687744, -0.838117778301239, 0.2846442759037018, -1.68349027633667, -0.6491971611976624, -0.9109901189804077, 1.2277722358703613, -0.037855762988328934, 0.17135195434093475, -1.4429761171340942, 0.5335408449172974, -0.23525185883045197, -1.167819857597351, 0.5230942368507385, 1.117510199546814, 0.18899978697299957, 0.5766450762748718, -0.8096867203712463, -0.573122501373291, -1.167766809463501, 1.7529593706130981, 0.11572638899087906, -0.1178494244813919, -1.634989857673645, -0.293171763420105, 1.9350677728652954, -0.8840469717979431, 0.24731381237506866, -0.7042083740234375, -1.0967864990234375, -0.3608889877796173, -1.1139559745788574, 0.8234314918518066, -0.46539396047592163, -1.2629979848861694, -1.449332594871521, 0.4433545470237732, -0.9642794728279114, 2.143775463104248, 0.7866349220275879, 2.1049644947052, -0.36800697445869446, -0.1784900277853012, -0.4202972948551178, 0.4548228085041046, 0.23048000037670135, -0.49779748916625977, 0.6772288084030151, 0.38781821727752686, -0.7906827926635742, -0.9614623188972473, -1.1853770017623901, 0.2015114575624466, -1.172881007194519, -1.571427345275879, 1.0257009267807007, 0.8765996098518372, -2.5580291748046875, -1.5568739175796509, -0.5165587663650513, -1.516141414642334, 0.15429694950580597, -0.7856951951980591, -0.4242229163646698, -1.0682685375213623, 0.18866883218288422, -1.064262866973877, 1.9229514598846436, -1.016007900238037, 0.5183945894241333, -0.5386597514152527, 0.1397111862897873, 2.4595532417297363, 0.018094073981046677, 0.22974592447280884, 0.5392224788665771, 0.22004668414592743, -0.23707804083824158, 0.08544835448265076, 0.10547315329313278, -0.3725636899471283, -0.19428232312202454, 0.09979801625013351, -0.4089242219924927, -0.2680543065071106, -1.8151487112045288, -0.8716955780982971, -0.4397434890270233, 0.8568159937858582, 0.8079643249511719, 0.2696060836315155, -0.5091552734375, 0.8732215762138367, 0.270706444978714, -0.10730698704719543, 0.5572503805160522, 0.31818968057632446, -0.8089329600334167, -0.7703138589859009, 0.4641139805316925, -0.7441592216491699, -1.5918076038360596, -2.3493599891662598, 0.41468289494514465, -0.011958830058574677, 0.8877599835395813, 1.1161266565322876, 1.8850184679031372, -0.03764907643198967, 0.61226886510849, 1.4469949007034302, 0.6065495014190674, -0.7310249209403992, 1.2785425186157227, -1.8094834089279175, 0.5715990662574768, -1.2691770792007446, 0.9491535425186157, 1.4857618808746338, -0.08456920087337494, 0.0927802175283432, -1.8269425630569458, -0.7421925663948059, 0.895503044128418, 0.9428671598434448, -1.2344459295272827, 0.8502984046936035, -0.40885409712791443, -0.46098142862319946, 0.04970614239573479, 0.8120710253715515, 0.8046832084655762, -1.4468512535095215, 0.7166003584861755, 0.15946997702121735, -1.611255407333374, 0.9613911509513855, 0.59262615442276, 0.23143711686134338, -0.7464098930358887, 0.34273234009742737, -0.5770589709281921, 1.2381129264831543, 0.08701223134994507, -0.9414792060852051, -0.485016405582428, -0.792411744594574, -0.670108437538147], [0.12203753739595413, 1.0842305421829224, 0.4219388961791992, -0.38340577483177185, -0.34270644187927246, 3.3215904235839844, -0.5376420617103577, -0.16260771453380585, -1.2990938425064087, 1.2626739740371704, -0.47920697927474976, -0.4518699049949646, 2.059223175048828, 2.124253511428833, 0.11555250734090805, -0.726203203201294, 1.0697845220565796, 2.3462066650390625, -0.8677769899368286, 0.27935048937797546, 0.10650308430194855, -1.3390446901321411, 0.9994052052497864, 1.1942808628082275, 2.6851422786712646, -0.770379364490509, -0.5024154782295227, -0.4828498959541321, 0.20915573835372925, 1.1190259456634521, 2.2913637161254883, 0.8908478021621704, 1.3166322708129883, 1.2049263715744019, -1.8033028841018677, -0.6813942193984985, -0.45439887046813965, 1.7150307893753052, -0.8508360981941223, 0.20235037803649902, 0.1935388445854187, -0.24814030528068542, 1.2042384147644043, 0.1753876954317093, 1.6728838682174683, -0.15437845885753632, 0.6399695873260498, -1.196351408958435, -0.8668498396873474, -0.5979136228561401, -1.113663673400879, 1.3906080722808838, -0.25778332352638245, -1.2807012796401978, 0.662202000617981, -0.3462282121181488, -0.13804200291633606, -0.7722307443618774, -0.9760934710502625, -0.3523181676864624, 0.9689339399337769, -0.704291045665741, -1.0334854125976562, -0.8141584992408752, 1.7812696695327759, -1.5264381170272827, 0.21901322901248932, 0.20984379947185516, -0.26635658740997314, 1.6966558694839478, -0.040970198810100555, -0.48329126834869385, -0.5936973690986633, 1.5450783967971802, 1.9672577381134033, -2.121931314468384, -0.7977190017700195, 0.653045117855072, -0.45130905508995056, -2.4832451343536377, 0.8577277064323425, 0.21530672907829285, 2.264127731323242, -0.35272273421287537, -0.8749279975891113, -0.6567586064338684, -0.6951943635940552, -0.29700008034706116, 1.8887617588043213, 0.7417891621589661, -0.15297652781009674, 0.4932132959365845, 0.19456855952739716, -0.7092868089675903, -1.8336833715438843, 0.6947963237762451, 0.11263292282819748, 0.5407277345657349, 0.42875587940216064, -0.1749887317419052, 1.133076548576355, 0.19008561968803406, 1.4020651578903198, 0.2553258538246155, 0.8619925379753113, -0.27250969409942627, -0.8066889047622681, 2.3385963439941406, 2.889517068862915, 0.27041158080101013, 0.06138103827834129, -0.818488359451294, 0.09643293172121048, -0.9763666987419128, -1.0333161354064941, -0.5255236625671387, -0.8124750256538391, -1.3312819004058838, -0.5728099942207336, 1.0325556993484497, -0.8281767964363098, -0.37798047065734863, -0.24118144810199738, 0.5039072036743164, -0.6523985862731934, 1.4229439496994019, -0.17012779414653778, -0.2261805385351181, 1.7079534530639648, -0.41244983673095703, 1.1438323259353638, -1.1403614282608032, 0.5060873627662659, 2.0272741317749023, 1.0472960472106934, 0.5608628392219543, 0.2145431488752365, -0.35900887846946716, -0.33917322754859924, 0.04676062986254692, 1.4573662281036377, -1.529097318649292, 0.5351682305335999, -0.41831809282302856, -1.5351486206054688, 1.5904085636138916, -0.32269635796546936, 1.5443487167358398, -0.6581053137779236, -0.3959701657295227, 0.3165018558502197, -0.10819648951292038, 0.14952684938907623, 0.5363402962684631, 0.28198134899139404, -2.7740068435668945, -0.9664573669433594, -0.49800199270248413, 1.5745774507522583, -2.4646682739257812, -0.7931223511695862, -2.2452025413513184, -0.6770046949386597, -0.5550649166107178, -0.49969571828842163, -1.2099148035049438, 1.9325053691864014, 0.23805038630962372, 0.020144658163189888, -0.07854729145765305, -1.5610538721084595, -1.2807537317276, 1.4219629764556885, -0.4186253249645233, 1.2185046672821045, -1.7696959972381592, 0.15892307460308075, 0.4078105390071869, 1.4724750518798828, 0.6121052503585815, -0.5980436205863953, -1.9627467393875122, 2.728254556655884, 0.042378172278404236, 0.4192182421684265, -0.22591879963874817, 0.6686848402023315, -0.6143389344215393, -1.090646505355835, 1.168336272239685, -1.3631681203842163, 0.23829969763755798, -0.8572589159011841, 0.21789708733558655, 2.578118324279785, 1.519744634628296, -1.2736269235610962, -0.041428737342357635, -0.5841810703277588, -0.3419249653816223, 1.0302938222885132, -0.17176058888435364, -1.0109511613845825, -0.23011082410812378, -0.02562272734940052, 0.577128529548645, -0.2517376244068146, 1.0330703258514404, -0.7457305192947388, -0.30782386660575867, 1.291614055633545, -0.4671914875507355, 0.346996009349823, 0.16923949122428894, 0.4831700026988983, -1.583005666732788, -0.7052438259124756, 1.241091012954712, -1.906534194946289, 0.440086305141449, -0.05184287577867508, 0.42164838314056396, -0.05808857083320618, -1.6964385509490967, 0.5508638620376587, -0.6106605529785156, -0.1889272779226303, -0.19573034346103668, -1.1147156953811646, 1.6633034944534302, -1.5080708265304565, -1.4788333177566528, -1.2340346574783325, -1.2839767932891846, -0.08859729021787643, -0.1299680918455124, 0.960803747177124, -0.6102650165557861, 1.4862632751464844, 0.727518618106842, 0.967659592628479, 0.6014463305473328, 0.05836217477917671, -0.7550533413887024, 1.1882537603378296, -0.2873777747154236, 0.10816120356321335, -1.121111512184143, 0.7643765211105347, 1.236053228378296, 0.820296049118042, -1.042913794517517, -1.6841915845870972, -1.0971611738204956, 0.48461657762527466, 0.21323390305042267, -1.2337323427200317, 1.3022905588150024, -1.2504042387008667, 1.1790876388549805, -1.694765567779541, 0.11102966219186783, 0.26192301511764526, 1.142791986465454, -1.4546968936920166, 1.9607844352722168, -0.6300467252731323, 1.6222842931747437, -0.4116450250148773, 2.036027431488037, 0.9031220078468323, -0.3039604425430298, 0.302591472864151, -1.3083893060684204, -0.9649621844291687, 1.010243535041809, 0.6995095014572144, -0.2878636419773102, 0.15383195877075195, 1.4302799701690674, 0.6851831078529358, -1.9687165021896362, -0.818699836730957, 1.3284659385681152, 0.6027844548225403, 0.7764161825180054, -2.009098768234253, -2.1259946823120117, -0.23232734203338623, -0.3063613176345825, 1.0579509735107422, 1.0070085525512695, -0.8107065558433533, -0.3491644263267517, 1.7445253133773804, 0.45821449160575867, 0.9241147637367249, 0.6819083094596863, 0.3496098518371582, 0.22272342443466187], [-0.9693304896354675, -0.0510660856962204, -1.552133560180664, 0.35545048117637634, -0.1403476744890213, 0.6823565363883972, -1.5722934007644653, -1.7902904748916626, -0.3083397448062897, 0.029081081971526146, -0.20707082748413086, -0.062272507697343826, -0.11533382534980774, -0.6122379899024963, 0.2579179108142853, 0.0318029560148716, 0.47448402643203735, -1.100888729095459, -1.2873908281326294, 0.5515273213386536, -0.31701794266700745, -1.0745689868927002, -0.4904130697250366, 0.3659607172012329, 0.19133830070495605, 0.11848250776529312, 2.234368324279785, 0.01268061250448227, 1.3117743730545044, 0.3909742534160614, -0.38515910506248474, -0.43788662552833557, 0.5825384855270386, 2.553140640258789, 0.8601570129394531, -0.23852337896823883, 1.7226852178573608, 0.6405743360519409, -1.2897918224334717, -2.493227481842041, -1.2851853370666504, -1.2259315252304077, 0.0969642698764801, 0.5548337697982788, 0.34736305475234985, 0.6303569674491882, 0.18612267076969147, 0.7399807572364807, 1.5716241598129272, -2.150848388671875, -0.665652334690094, 1.6830936670303345, 2.4097678661346436, -0.028255591168999672, -0.8171101212501526, 0.5572949051856995, -0.17531266808509827, -0.8256411552429199, 0.11928264796733856, -0.0757257342338562, 0.33273136615753174, 5.0168986490461975e-05, 0.5246039032936096, -1.274179458618164, -1.590874433517456, 0.40788087248802185, -2.2328248023986816, -1.8542838096618652, -0.767042875289917, 0.3671681582927704, -0.9433631896972656, -1.2865620851516724, -1.372411847114563, 0.21535645425319672, 1.936025857925415, 0.8779425621032715, 0.35428041219711304, 1.7431343793869019, 0.8605430126190186, -0.6535939574241638, -1.442739725112915, -0.42570021748542786, -0.37755972146987915, -0.47284990549087524, 2.1820931434631348, -0.037088606506586075, -0.9145176410675049, -0.6947476267814636, 0.4887133240699768, -0.10955149680376053, -0.5525664687156677, 0.147037073969841, -0.4908849895000458, 0.7429641485214233, 0.21958191692829132, 0.45688292384147644, -0.0511883907020092, -0.5856831073760986, -0.0878090038895607, -0.091180220246315, -0.2619696855545044, -1.4796454906463623, -1.4952821731567383, -0.3490339517593384, 0.9525327086448669, 1.7959762811660767, -0.3332998752593994, -1.3451499938964844, -2.5946948528289795, -0.703977644443512, 1.224119782447815, -0.2754623293876648, 1.425851821899414, -0.5444273352622986, 1.6215946674346924, 0.7781478762626648, 0.28502750396728516, -0.4800873100757599, 0.19002683460712433, 0.1413576751947403, -0.4187442660331726, 0.28088903427124023, -0.5826550126075745, 0.08445700258016586, -0.3818908631801605, -1.4819427728652954, 1.8428380489349365, 0.41313961148262024, -2.1395065784454346, 1.0463128089904785, 0.7903568148612976, 0.05142604187130928, -1.1457712650299072, -0.32849618792533875, 0.8698682188987732, -0.1672412008047104, -0.9642161130905151, -1.081983208656311, -0.5485547780990601, -3.1006975173950195, -0.6445205211639404, -0.0674225389957428, 1.0402687788009644, 0.4537600874900818, 1.4156017303466797, 0.03192668408155441, 0.7443245649337769, -0.5114455223083496, -0.9304673671722412, -0.167079895734787, 0.7104073166847229, -0.6855648756027222, -1.315915822982788, 1.1757943630218506, -0.23405760526657104, -0.03614093363285065, 2.439826726913452, 0.19731366634368896, 0.2581629753112793, 0.00867868959903717, 0.826564371585846, 0.35991236567497253, -0.5690733790397644, 0.07230637967586517, 0.9003498554229736, -1.5349600315093994, -0.4370964467525482, 0.7923745512962341, -0.532792866230011, -0.5223993062973022, -0.41775426268577576, 0.13005198538303375, 0.4622197151184082, -0.13313527405261993, -0.3131673336029053, -0.9563115239143372, -1.4292160272598267, -1.4520257711410522, -2.451650619506836, 0.816656231880188, 0.3676561713218689, 0.09034666419029236, -0.16732603311538696, -1.204468011856079, 2.398477554321289, 1.6682780981063843, -0.34782132506370544, -0.26698464155197144, 0.6471540927886963, 0.4862140715122223, -0.8134186863899231, -1.6686339378356934, -0.757193922996521, -0.5435447692871094, -1.1271982192993164, -1.5523782968521118, -0.4198416769504547, -0.2113131731748581, -0.5565385222434998, 1.834326982498169, -0.6959983110427856, -0.47204387187957764, -0.2744533121585846, 0.07088450342416763, 0.4426359236240387, 0.6262314915657043, -0.6699908375740051, -0.11693315207958221, -0.11315543204545975, -1.6173522472381592, -0.7865963578224182, -0.7732769846916199, 0.7880830764770508, -0.14301364123821259, -0.8548619151115417, -1.1589879989624023, 0.5342168807983398, 1.6192713975906372, -1.8025201559066772, -0.40450844168663025, -0.44782739877700806, -0.7103159427642822, 0.49233564734458923, 0.6131160855293274, 0.595209002494812, -0.26219892501831055, 0.10909741371870041, 0.6263179779052734, -0.04607170820236206, -0.806883692741394, -0.5919789671897888, -0.039696309715509415, 0.6873367428779602, 1.5253487825393677, -0.5346082448959351, 0.7449365258216858, -0.2846798598766327, 2.9385793209075928, -0.6633299589157104, 1.297149896621704, -1.9794756174087524, 1.736634373664856, 1.8862814903259277, -1.597500205039978, 1.136823058128357, -0.14437399804592133, -1.8392597436904907, 0.2457991987466812, 0.02552398107945919, 0.693006694316864, -2.696650266647339, -2.257481336593628, -0.09542977064847946, -1.051011562347412, -0.3294304609298706, 0.02570136822760105, 0.954003095626831, 0.48858389258384705, 1.2771637439727783, 1.0218671560287476, -0.13628053665161133, -0.0872027575969696, -0.8175540566444397, -0.13591375946998596, 0.23243214190006256, 0.07158955186605453, 0.6092890501022339, 0.009168412536382675, 0.9387616515159607, -0.2886286675930023, 0.6669642925262451, 0.8662517070770264, -0.10945890098810196, -0.3731740117073059, -0.2892800569534302, -2.330291271209717, -2.2785420417785645, 0.05000414699316025, -1.7506115436553955, 0.2864363193511963, -1.1850172281265259, -1.5913296937942505, -0.23406045138835907, -1.4983575344085693, 0.67307049036026, -0.4675431549549103, -0.41563552618026733, 0.6790832877159119, 0.8266547322273254, -1.5817294120788574, 0.958221971988678, -1.2697322368621826, 1.4557887315750122, -0.804319441318512, 1.7232521772384644, 0.634728193283081, 1.3755532503128052, 0.801731288433075, 0.465299516916275, 1.1576205492019653], [0.21499764919281006, 0.37600263953208923, -0.24216797947883606, -0.17356830835342407, 0.14852288365364075, 0.7579862475395203, 2.1016671657562256, 2.1470987796783447, -1.2401455640792847, 0.9423704147338867, 0.800032913684845, -2.0414719581604004, 1.1009056568145752, -0.7270483374595642, 0.7182489037513733, 0.4480334520339966, 0.6246030926704407, 0.5375230312347412, 1.324334979057312, -0.15445104241371155, 2.155627727508545, -0.28463757038116455, 0.5468308329582214, 0.059071097522974014, 0.010513871908187866, 0.9682322144508362, -0.1916487216949463, 0.23056434094905853, -1.4556044340133667, 1.9453643560409546, 0.895625114440918, 0.9649550914764404, -0.28162863850593567, 0.1345972716808319, 0.9429124593734741, 1.3317577838897705, -1.3129034042358398, 0.13933591544628143, 0.9637857675552368, 1.242737889289856, 1.1391185522079468, 0.3601077198982239, 0.9136673212051392, 0.2962907552719116, 0.5166333317756653, 0.0703456774353981, 0.6604983806610107, 0.35157516598701477, -1.2056617736816406, -0.24036866426467896, -0.48085322976112366, 0.35286155343055725, -0.3462119996547699, -0.6421433091163635, 2.6769113540649414, 1.5738331079483032, -1.7250558137893677, 1.3278257846832275, 0.16028393805027008, 0.48969000577926636, 1.7442374229431152, -0.1200537234544754, -0.4983961284160614, -1.3688364028930664, -0.870997965335846, 0.6896116137504578, 0.7640365362167358, 0.9673698544502258, 1.7797945737838745, -0.09931761771440506, 0.40624505281448364, -0.48390141129493713, 1.4123362302780151, -0.8182118535041809, -0.5887675881385803, 0.3052622079849243, -1.568239688873291, 0.7000134587287903, 0.5331606268882751, -0.8048045635223389, 0.7904122471809387, -1.5673998594284058, 1.6082124710083008, 1.1531620025634766, 0.0486750490963459, -1.5507760047912598, -0.9743438363075256, -0.6336786150932312, 0.563026487827301, 0.17016927897930145, -0.33218270540237427, -1.1333671808242798, -2.1527202129364014, 0.31693586707115173, -1.2188023328781128, 1.1472315788269043, -0.420636922121048, 1.18535315990448, -0.8467777967453003, -1.133001446723938, 0.3117910325527191, -0.01974562741816044, -0.22805245220661163, -0.6175290942192078, 0.637578547000885, 1.7955931425094604, -0.6490527987480164, -1.1511964797973633, -1.3164286613464355, 0.8029506206512451, 0.3003692924976349, 0.42696499824523926, -0.08976663649082184, -0.9123473763465881, 2.3397812843322754, 0.3998563587665558, 0.8392680287361145, 0.6872902512550354, -0.9884225130081177, 1.1613517999649048, -0.823573112487793, 1.4557158946990967, 1.3023343086242676, 1.695466160774231, 1.9652214050292969, -1.1092549562454224, 0.7957472801208496, 0.3994406461715698, 0.27412113547325134, 0.9967520833015442, 0.6294512748718262, 0.45432978868484497, -0.03702706843614578, -0.2708207666873932, 0.20526240766048431, 0.9227113723754883, -1.0934737920761108, 3.4550940990448, -0.08439964056015015, -0.7798764705657959, -1.0755451917648315, -0.19100293517112732, -0.49639901518821716, -0.5289570689201355, -0.6600319147109985, 0.984561562538147, -0.2615361511707306, 0.5889118909835815, 0.3483147621154785, -0.24397364258766174, -1.829586148262024, 0.8510041832923889, 0.04836748540401459, 1.0049045085906982, -0.6311933994293213, -1.1768327951431274, -0.4810082018375397, -0.47101467847824097, 3.3464159965515137, 0.39700889587402344, 1.8445236682891846, 0.37368571758270264, 0.4403616189956665, 1.5881816148757935, 0.8414890170097351, -0.33890190720558167, 1.9745346307754517, 0.4584785997867584, 0.5126276016235352, 1.6238698959350586, -2.575951099395752, 0.10392799973487854, -2.383230447769165, 0.2029256820678711, -1.0326638221740723, -1.5759837627410889, -1.6860159635543823, -0.2958468794822693, -0.4653347432613373, -1.2779077291488647, -1.4443118572235107, 2.092310905456543, 1.53140127658844, -0.20553794503211975, 1.2940083742141724, 1.4425323009490967, -0.34261560440063477, -0.6088219881057739, -0.9251545071601868, 1.5151461362838745, 1.3861082792282104, -1.5262259244918823, 0.049991317093372345, -0.11324882507324219, -1.2685106992721558, 1.8059941530227661, 0.9912446141242981, -2.0469563007354736, -0.17157728970050812, 0.607613742351532, 0.912075400352478, -2.1449060440063477, 1.725726842880249, 1.476056694984436, -0.6266317963600159, 0.6996612548828125, 0.025268597528338432, -0.6298747658729553, 1.2111462354660034, -0.2649790644645691, -1.1712124347686768, -0.15496279299259186, 0.2631274461746216, -0.4257051348686218, -0.3237447142601013, 0.8859488368034363, 1.1502861976623535, -2.8515398502349854, 1.1822530031204224, 0.48114338517189026, 0.18479636311531067, 0.6750312447547913, 0.6610159873962402, 0.5793243050575256, -0.3437846899032593, -1.7133105993270874, 1.4399269819259644, -0.5063510537147522, 0.3406410813331604, -0.9866684675216675, -1.7020512819290161, -0.13344340026378632, 0.5084496140480042, 0.28555434942245483, 0.2575523257255554, 0.3876165449619293, 1.8681575059890747, -0.9719213247299194, -0.5167950987815857, -0.5564690828323364, 0.5244477987289429, 1.5884944200515747, -1.433326005935669, 0.6646203994750977, 0.7627796530723572, -0.10773871839046478, -1.0622777938842773, 1.0591456890106201, 0.05268985033035278, -1.786608099937439, -0.8904621005058289, 1.1511021852493286, 0.7231478095054626, -1.9836010932922363, -2.244274616241455, 0.2930326759815216, 1.2209243774414062, -0.7010833621025085, -1.0314146280288696, 0.8255687952041626, 0.07029740512371063, 0.746677815914154, -1.3378212451934814, -0.39617106318473816, 1.0964717864990234, 0.443949818611145, 1.1159381866455078, -1.5842819213867188, -1.7577948570251465, -0.7328300476074219, 0.8811623454093933, -0.262769490480423, 0.3437105715274811, -1.2605576515197754, 1.0253885984420776, 0.09812329709529877, -0.7820639610290527, 0.21945340931415558, -1.2617689371109009, -1.6670750379562378, 0.3601551651954651, -0.5909396409988403, 0.1471080780029297, 0.19821450114250183, 1.4128572940826416, 1.1634315252304077, -0.719045877456665, 1.710498332977295, 1.1599643230438232, -0.02437640354037285, 0.192474365234375, 0.24381670355796814, -0.05801231786608696, 0.16741704940795898, -2.1164119243621826, -1.2506779432296753, -0.18238162994384766, -0.07414279878139496, -0.6659558415412903, -1.0648454427719116], [-1.2741743326187134, 0.6433325409889221, -0.09134025871753693, 1.0724290609359741, 0.7096996903419495, 0.4402373731136322, 0.5845980644226074, 2.236281633377075, 0.11840242892503738, -0.7260212302207947, 1.034353494644165, -1.45391047000885, -0.8359228372573853, -1.1106045246124268, -0.5371758937835693, 0.7039821743965149, 0.1502380520105362, 0.7329959869384766, -0.46418675780296326, -1.615173101425171, 1.3111525774002075, -1.5598756074905396, 1.308302640914917, -0.6847361326217651, -0.8128980398178101, 1.5278903245925903, 0.053538743406534195, -0.4104577302932739, -0.672728419303894, -0.003119724802672863, -0.8146275281906128, -1.4267607927322388, 0.45378538966178894, 0.9572137594223022, 0.007141768932342529, 0.6780993938446045, -0.4052402675151825, 0.06925646215677261, 0.07715467363595963, -2.0104165077209473, -0.7116667628288269, -0.01185669656842947, -1.1543967723846436, 1.0682857036590576, -0.7925015091896057, -1.4530410766601562, -0.2756928503513336, 2.043531894683838, 0.42490091919898987, 0.22190995514392853, 0.3491470217704773, -1.1948466300964355, -0.5318862795829773, 0.20285920798778534, 0.3353745639324188, -0.5576741099357605, 2.820990800857544, -0.5259838104248047, -0.7698396444320679, 0.2322106659412384, 1.1025317907333374, 0.5343267321586609, 0.6672408580780029, 0.49864187836647034, -0.13563476502895355, -0.9980189800262451, -1.1321109533309937, 0.8061579465866089, 1.2844911813735962, 1.5339529514312744, -1.524580955505371, 0.3398969769477844, 1.6199557781219482, 0.9083226919174194, 1.555924654006958, -1.881919026374817, -0.42567694187164307, -0.6908568143844604, 1.0000735521316528, -0.37845975160598755, -0.21123944222927094, 2.3795340061187744, 2.2670974731445312, 0.2919250428676605, -0.43650346994400024, 1.6165851354599, 1.4089555740356445, 0.6749752163887024, -0.6197512745857239, 0.1044793576002121, 2.3328440189361572, -0.30156588554382324, 1.2007485628128052, 0.5241599678993225, -1.683212399482727, -0.8821189999580383, 0.7056115865707397, -0.9282427430152893, 1.0341764688491821, -1.5973176956176758, 1.0011857748031616, 0.6734509468078613, 0.7722969055175781, 0.3619694411754608, -1.3134456872940063, -0.4408937394618988, -0.0903163030743599, 1.3246560096740723, -1.2582675218582153, -0.796372652053833, 1.2139148712158203, 1.5735715627670288, -0.6206997036933899, -2.5162270069122314, 0.7143406867980957, -0.06834883242845535, -0.3927680253982544, -0.34026432037353516, -0.9715268611907959, 0.059322163462638855, -0.057333577424287796, -0.6256800889968872, -1.1233608722686768, -0.2292230725288391, -1.4657490253448486, -1.195326566696167, 0.1089729368686676, -1.4597934484481812, 1.7337286472320557, 0.559247612953186, -0.31334665417671204, -0.313558965921402, 1.9379969835281372, 1.4422858953475952, 1.996575951576233, 1.2192580699920654, -0.6556313633918762, 0.4044710397720337, 0.38495272397994995, -2.2547953128814697, -0.5929520130157471, 0.03974044322967529, -0.04799430072307587, 1.438448190689087, -0.5718445181846619, 0.039138760417699814, -0.0016010106774047017, 0.11664614081382751, -1.0749105215072632, 0.6955568194389343, 1.0702499151229858, -1.1375328302383423, -1.527387022972107, 0.16391275823116302, -1.3779573440551758, -0.16977407038211823, -0.05545378476381302, 0.10578159987926483, -0.0651431679725647, -1.1839311122894287, 1.0826326608657837, 1.3657793998718262, 0.18640190362930298, 0.017485663294792175, 0.4513765871524811, -0.6050324440002441, 2.128770112991333, -1.003185510635376, -0.7019733190536499, 2.127373695373535, 0.8909721374511719, 0.3267163336277008, 0.6558904647827148, 0.24737748503684998, -1.3520294427871704, -1.1243493556976318, 0.3582913279533386, 0.29746392369270325, 0.716314971446991, 0.9811283349990845, -0.6384065747261047, -0.2767465114593506, -1.1906055212020874, -1.4136744737625122, 1.0638657808303833, -0.5086575150489807, -0.9359400868415833, -0.21788711845874786, 1.5416470766067505, -0.5921344757080078, 0.6777596473693848, 1.3148833513259888, 0.9549416899681091, 1.1057815551757812, 1.1407060623168945, -0.6113197207450867, 1.9605149030685425, -0.4782751798629761, -1.416520357131958, -0.5977271199226379, 1.4953269958496094, 0.43994519114494324, -1.4995708465576172, 1.1197782754898071, 1.5765794515609741, -0.1186833530664444, 0.5912532806396484, 0.211544468998909, 1.3161859512329102, 0.5927997827529907, 1.6817984580993652, -0.546570897102356, -2.4889259338378906, -2.093226909637451, -2.0605955123901367, 0.9286244511604309, -0.0671006366610527, -0.7165977954864502, 0.25169336795806885, 0.36750707030296326, 0.9476193785667419, 0.728239893913269, 0.11825525760650635, -0.3175182640552521, -0.42063117027282715, 0.8313623070716858, -1.2932755947113037, -0.48271504044532776, -0.29503944516181946, -0.398101806640625, -0.07409965991973877, 0.2775367200374603, -0.334458589553833, -0.6559234261512756, -0.9679246544837952, -0.8800956010818481, -0.09278975427150726, 1.8405507802963257, -1.0398329496383667, 0.18076138198375702, -0.5975000858306885, -0.4454463720321655, -0.08544299006462097, -0.0033291056752204895, 1.795460820198059, 0.3505346477031708, -0.21249021589756012, 0.2479475736618042, -0.5524846911430359, 0.5404183864593506, 0.4447035789489746, -1.303989052772522, -1.176607608795166, 2.050729513168335, -0.5857805609703064, -0.24929285049438477, -0.2863468527793884, -0.2744353115558624, 0.6172606945037842, -1.8251465559005737, 1.4828829765319824, 0.6355683207511902, 0.609643280506134, -0.4574022591114044, 1.4711554050445557, 0.4363490343093872, 1.226547122001648, 1.0419734716415405, -0.24362117052078247, -1.4806832075119019, -0.8299579620361328, -0.04686404764652252, -0.31384700536727905, -0.6466959714889526, 1.6789557933807373, -1.5062434673309326, -0.8494601845741272, 0.5944042205810547, 1.3895785808563232, -0.4241028428077698, -0.2964038550853729, -0.18299630284309387, 0.4109027683734894, -0.06776145845651627, 0.6839215755462646, 0.3707437813282013, -2.0952019691467285, 0.2505626976490021, 0.5341809988021851, -0.7668609619140625, 0.8019914031028748, 0.3801518976688385, 0.6575012803077698, -0.0732053741812706, 0.5574610233306885, 0.7225639224052429, 2.146333932876587, -0.09299785643815994, 0.5109341144561768, 0.12435397505760193], [1.1026979684829712, -0.9454220533370972, -1.3643532991409302, 0.4272956848144531, -0.4910997748374939, -1.5126523971557617, -1.3566962480545044, 2.08284330368042, -0.22962817549705505, -1.1906613111495972, -0.03976305574178696, -0.3690812289714813, -0.5732136964797974, 0.44229787588119507, -1.6968727111816406, -2.198690176010132, 0.1409667432308197, 0.35461756587028503, 0.5743932723999023, -1.0320204496383667, 0.11837435513734818, 0.5048470497131348, -0.5674140453338623, -2.0369160175323486, 0.5030789375305176, 0.6260127425193787, -0.18173375725746155, 0.6105136871337891, -0.016361156478524208, 0.8617801070213318, 0.21815474331378937, 1.3094205856323242, -0.5527933835983276, -0.1378438025712967, -0.13814590871334076, 1.3723196983337402, -1.7921088933944702, 0.01774604618549347, 0.7324461340904236, 0.42485311627388, -1.2135839462280273, -0.9990777373313904, -0.4392493665218353, 0.7163751125335693, 0.8392165899276733, 0.820513904094696, -0.020930161699652672, 0.5354667901992798, 0.17876975238323212, 1.7486319541931152, 0.5895655155181885, 0.8570089340209961, 0.6822199821472168, 0.29165688157081604, 0.24994772672653198, -0.013928000815212727, -0.8541018962860107, -0.11085585504770279, -0.41237545013427734, 0.3985804617404938, 0.4567168653011322, -0.8432623147964478, -1.472763180732727, -0.07528293132781982, -0.7771479487419128, -0.6923969388008118, 0.30477648973464966, -0.4984729588031769, -2.511305093765259, -0.6079760193824768, -0.4326394498348236, 0.08448932319879532, -0.12379226088523865, 0.6939185857772827, 0.04450007528066635, -0.2966713309288025, 0.5828737020492554, -0.4689366817474365, -1.30525803565979, -0.6115255951881409, -0.9924113154411316, 0.9780524373054504, -0.7746453881263733, 0.5583546161651611, 0.6253305077552795, -0.6967886090278625, -0.5145695805549622, 0.37018853425979614, -1.1803898811340332, -0.035627562552690506, -0.5068316459655762, 0.05343121290206909, -0.9217963218688965, 0.5217757821083069, -1.7288986444473267, 0.193372905254364, 1.8765934705734253, 1.124350666999817, -0.045101456344127655, 0.4906404912471771, 0.7136102914810181, -0.06286304444074631, 1.249509572982788, 0.5820499062538147, -0.18203507363796234, -0.5780484676361084, -0.4363299012184143, 1.2301275730133057, 1.5449721813201904, -1.1601415872573853, -0.22401605546474457, -2.3446948528289795, -0.36221712827682495, 1.1004308462142944, 0.8761981725692749, 1.1453235149383545, -0.7596340775489807, -0.12784941494464874, 2.0772275924682617, 0.20797030627727509, 0.4302642047405243, 0.9202303886413574, 1.1016013622283936, 0.029562782496213913, -0.6703923344612122, -0.9143137335777283, 0.569887638092041, 0.29211556911468506, 2.3085436820983887, -0.13395918905735016, 0.2926991283893585, 0.2013433873653412, -1.1255923509597778, 0.9450353384017944, 0.3240194022655487, -0.5603557229042053, 0.24655914306640625, 1.4876506328582764, 1.2093912363052368, 0.8708429336547852, 1.3027503490447998, -0.1724250167608261, 0.8604784607887268, 0.10453595221042633, 0.3069404661655426, -0.2821875214576721, 1.549726128578186, 0.9967539310455322, -0.7965542078018188, -0.39137959480285645, -0.3419218361377716, -0.5477572083473206, 0.45690178871154785, 0.3836056888103485, 0.8579692840576172, -0.6274529099464417, -0.21587321162223816, -0.13204389810562134, -0.9321666359901428, 1.312090277671814, 0.018624402582645416, 0.11027354747056961, 0.6749672889709473, -0.04273483529686928, -1.129655361175537, 1.1166434288024902, 1.1344108581542969, -0.40572652220726013, 1.1515129804611206, -0.32521119713783264, 0.48745197057724, -1.7346822023391724, 0.3317869007587433, 1.7549633979797363, 1.6187242269515991, -0.6857925057411194, -1.03154456615448, -0.7229616641998291, 0.09434501826763153, -0.18558548390865326, -0.19016198813915253, -1.7168139219284058, -0.6732166409492493, 0.8242234587669373, -0.8808163404464722, 0.4011859893798828, 0.41987884044647217, 1.304154396057129, 0.5157730579376221, 0.20882350206375122, -0.10567475855350494, 0.7058365345001221, -1.284270167350769, -0.8750055432319641, -0.1278020590543747, 1.1691651344299316, 0.33011603355407715, -0.404538094997406, -0.21876342594623566, -0.09955651313066483, 1.6720972061157227, 0.9537591934204102, -0.31405073404312134, -1.0663775205612183, -0.5028542280197144, -0.9342756867408752, 0.7726858854293823, 0.18164055049419403, 0.4980334937572479, -0.42937135696411133, 0.40306445956230164, -0.2256525456905365, 1.1445591449737549, -0.5831671953201294, 0.07620875537395477, -0.16819554567337036, -0.16962747275829315, -0.5123867988586426, -0.35116052627563477, -0.8930013179779053, 0.7672109603881836, -0.9221373200416565, 0.1349612921476364, 0.8197247982025146, -0.7467048764228821, -2.8480474948883057, 0.4542182981967926, 0.11214965581893921, 1.6600645780563354, -0.7861265540122986, -0.9107900261878967, -0.3909724950790405, -2.052539348602295, -0.27438458800315857, -0.3352378308773041, 0.11611674726009369, -0.9284414052963257, -0.473364919424057, 0.25846201181411743, 0.6373090147972107, -1.6417605876922607, -0.6910932660102844, 1.6233952045440674, 0.8126107454299927, 0.9376245141029358, -0.936733603477478, 2.214315891265869, -1.6110657453536987, 0.48813360929489136, 0.9423316717147827, -0.9280811548233032, -0.4319654703140259, 0.13979461789131165, 0.7494121789932251, 0.12678328156471252, 1.665344476699829, 0.7829982042312622, 1.8618278503417969, 1.082939624786377, 1.3806726932525635, -0.35072755813598633, 1.1511121988296509, -0.4962303638458252, -1.1237844228744507, -0.4570021331310272, -0.0490570142865181, 1.8869463205337524, -0.7176320552825928, 0.2920861542224884, -0.8330960273742676, 0.7410358786582947, -0.5191198587417603, -0.6694623231887817, 0.4794301390647888, 0.055083177983760834, -0.9281794428825378, 0.06419120728969574, 1.797768473625183, 0.25831422209739685, -1.2545713186264038, -1.9639402627944946, -1.3070677518844604, -2.2088820934295654, 1.5520155429840088, -1.3084819316864014, -0.6487635970115662, -0.6758741140365601, 0.16844452917575836, -3.254290819168091, -1.5853108167648315, -0.6997907757759094, 0.8517835736274719, -0.9881733059883118, 0.2889907956123352, 0.14412935078144073, 1.7712111473083496, -1.9437360763549805, -1.1184364557266235, 0.06177535653114319, 0.2188255935907364], [-1.366979956626892, -0.8063830733299255, 1.325127124786377, -2.3537189960479736, 0.8991081118583679, -0.15561647713184357, 0.5886068344116211, -0.0868070125579834, -0.3906925618648529, 0.1620136797428131, 1.518595576286316, -0.014428368769586086, -1.1905372142791748, 0.44384586811065674, 0.9393037557601929, -0.502524197101593, 1.2271519899368286, -1.0183191299438477, 1.0662956237792969, -0.6779590249061584, 1.0106769800186157, 0.8100352883338928, 2.3131375312805176, 0.6087737679481506, 0.13098716735839844, 1.1092170476913452, 0.3307836651802063, 2.591703414916992, 0.16139915585517883, 0.4262394905090332, 1.8472530841827393, -0.17281495034694672, -1.7770332098007202, 0.19666609168052673, 0.3139594793319702, 1.1179920434951782, 0.6963827610015869, -1.7739391326904297, 0.6049245595932007, 0.5883256196975708, -0.9593233466148376, -1.1828633546829224, 0.0965215265750885, -0.20226044952869415, -2.007080316543579, 0.9507967829704285, -0.20064125955104828, -0.9239527583122253, -2.0941123962402344, -0.23607294261455536, -0.20228107273578644, 0.7645199298858643, -1.0302798748016357, 1.3670992851257324, -0.6071594953536987, -0.2992483079433441, -0.2846405506134033, 2.31097412109375, 1.1676090955734253, -0.04826515540480614, 1.260711908340454, -0.3619833290576935, 2.282686948776245, 0.15170252323150635, 0.9799023270606995, -1.6075550317764282, -0.48511794209480286, 1.2570523023605347, 0.5065543055534363, -0.4030733108520508, -0.14325478672981262, -0.9016414880752563, -2.0610032081604004, -0.5198498964309692, -1.608223795890808, 0.24295011162757874, -0.2909548282623291, 0.027581872418522835, 0.9950820207595825, 0.3388361930847168, -1.5066086053848267, 0.979255199432373, -1.5878832340240479, 0.12162228673696518, -0.7406481504440308, 0.5181358456611633, -1.3199456930160522, 1.0344550609588623, -1.3986104726791382, 1.008157730102539, -1.0067620277404785, 1.7240420579910278, 1.3988484144210815, -0.8334418535232544, 1.0987322330474854, 1.6115975379943848, -0.5870162844657898, 0.003237728262320161, -1.0569560527801514, -2.1355371475219727, -0.4743054211139679, -0.37053704261779785, 1.8146841526031494, -0.5470268130302429, 1.3026009798049927, -0.07862956076860428, 0.7850841879844666, -1.2475061416625977, -0.15947219729423523, 0.348073273897171, 0.07383642345666885, 0.2896837890148163, 0.6353414058685303, 0.030179863795638084, 1.556154727935791, -0.06829964369535446, 0.4993179738521576, -0.6108224391937256, -0.33375272154808044, 1.5007184743881226, 0.7578912973403931, 0.18859468400478363, 1.232627034187317, 0.883998692035675, -0.022832432761788368, -1.0338042974472046, -0.16001874208450317, -0.3630172312259674, -0.14561627805233002, 0.9797621965408325, 0.3870580196380615, 0.2582855224609375, 1.0333850383758545, 0.08842363208532333, -1.2524067163467407, 0.6516528725624084, 0.27249395847320557, 0.29049962759017944, -1.5787057876586914, 1.3049017190933228, 0.3545279800891876, -0.9128248691558838, 0.9353909492492676, -0.2893674969673157, -1.156463861465454, 0.3401638865470886, -0.4276403486728668, -0.02881711907684803, 0.43268564343452454, -0.45884448289871216, -0.6444478034973145, 2.0254507064819336, -0.7498482465744019, 1.146978735923767, -0.3670909106731415, -0.4638573229312897, -0.4249015152454376, -0.21282006800174713, -1.9581447839736938, -0.022354599088430405, 1.9755339622497559, -0.15865251421928406, -0.01734325662255287, 1.3958364725112915, 0.9085615873336792, -1.0061085224151611, -1.6392426490783691, -0.45395198464393616, 0.040985286235809326, 0.024226348847150803, 0.39326614141464233, -0.5501708388328552, 0.04611360654234886, -1.5474804639816284, -0.593377947807312, -0.5286551713943481, 1.790993332862854, 0.7188053131103516, -1.0563445091247559, -1.0580147504806519, 0.7040716409683228, -1.3764139413833618, 1.3092595338821411, -0.5341575145721436, 0.4154508113861084, 1.928698182106018, 0.39414504170417786, 0.2619144916534424, -1.298645257949829, 0.8337897658348083, -0.3609415292739868, -1.4313312768936157, 0.6666193008422852, -0.21864940226078033, 1.2110203504562378, 0.6875128149986267, -0.07017832249403, -0.270539790391922, -0.36613571643829346, -0.45761755108833313, 0.5316331386566162, -0.5137923359870911, 1.1975678205490112, -1.3715850114822388, 0.13898496329784393, 0.4054080545902252, -0.11795683205127716, -0.6227046251296997, -0.33095771074295044, -0.7122765183448792, -1.7710541486740112, 0.2731650173664093, -1.3756499290466309, -1.5892055034637451, 0.5660121440887451, 0.2178051620721817, -0.2725588083267212, 0.32941895723342896, -0.3369474411010742, -0.08001662790775299, 0.1523703634738922, 0.8066645860671997, 0.20201009511947632, -0.3931635916233063, 1.243378758430481, 0.32871729135513306, 0.14029426872730255, 1.1547633409500122, 0.6691802740097046, -0.639841616153717, -0.9718174934387207, -0.9664644002914429, -0.46200209856033325, -0.5762079358100891, 0.2611842453479767, 0.07619637995958328, 1.2806453704833984, -0.6770594120025635, -1.5004780292510986, -2.013988733291626, 0.7058389186859131, 2.3103396892547607, 0.5732089281082153, -0.941361129283905, -0.22829923033714294, -0.2511429786682129, -2.520531177520752, -0.03991009294986725, 0.8419223427772522, 0.6018251776695251, 0.14791657030582428, -0.4945991337299347, 0.3543381989002228, -0.7830525040626526, 1.5739223957061768, 0.6525411605834961, 1.3037053346633911, 0.6500127911567688, 0.5416824221611023, -0.2068025767803192, -1.5539629459381104, 4.044939994812012, -1.2952719926834106, -0.09325821697711945, -0.65703946352005, -0.5740150213241577, -0.4136052429676056, 0.9872438311576843, -0.41136571764945984, 0.6714783310890198, 0.9183269143104553, -0.2822457253932953, -0.09426417946815491, 0.07723968476057053, 1.29778254032135, 0.19967949390411377, -0.5275925993919373, -0.15828315913677216, 1.9878615140914917, -0.5496024489402771, -1.4321026802062988, 0.010205188766121864, 0.5555524230003357, -0.06108551472425461, 0.16955627501010895, 0.9929311275482178, -0.8280360698699951, -0.2386569231748581, -1.006854772567749, -1.5423104763031006, 0.9904958009719849, 2.1162047386169434, -0.019143536686897278, -0.5245453119277954, 0.8772926926612854, -1.7508898973464966, -0.5932966470718384, -1.500874638557434, -0.17622767388820648, -0.7211750745773315], [-1.9397423267364502, -1.1730003356933594, 0.6692510843276978, 0.1810360848903656, -1.1633000373840332, 0.3793904483318329, 0.647479772567749, 0.26462772488594055, 0.4473758637905121, 0.40257522463798523, 1.6892863512039185, 0.02432568557560444, 0.36635100841522217, 0.05465323477983475, -0.5451669096946716, -1.117486596107483, 1.4210237264633179, -0.5389423966407776, -1.3045109510421753, -1.4825832843780518, -0.6946315169334412, 1.225628137588501, -0.10431496798992157, -0.7732913494110107, 0.6787874698638916, -1.499228596687317, 0.7829306721687317, 0.28855833411216736, 0.4123367965221405, 0.3695700168609619, 2.301180362701416, -1.3907397985458374, 0.23048652708530426, -0.37176841497421265, 0.8017458319664001, -1.0922311544418335, 0.4413672685623169, -1.852441430091858, 0.21613329648971558, -0.5472002625465393, -0.5208404660224915, 0.2768075168132782, -1.584889531135559, 0.3280569314956665, 0.5473589897155762, 1.8843610286712646, 1.2370493412017822, 0.8662002682685852, -1.1948448419570923, -1.06206476688385, -0.9161115884780884, -0.48507726192474365, -1.3291958570480347, -0.330521821975708, 0.948041558265686, -2.493187189102173, 0.025058524683117867, 0.1804015189409256, 1.467334270477295, 0.02681625261902809, 0.6315444707870483, 1.2283686399459839, 1.7809323072433472, 0.5905596017837524, -0.6423230767250061, -0.3758145868778229, -0.3545927107334137, 0.9700030088424683, 0.6983941793441772, 0.08183273673057556, -0.8373090028762817, 0.2171536684036255, 0.17388825118541718, -0.21174168586730957, -0.6058590412139893, -2.0176970958709717, -1.7179938554763794, 0.9258832335472107, -0.16687852144241333, 1.7744522094726562, 0.8257582783699036, -1.7905837297439575, -0.9438486695289612, 0.180597722530365, -0.7794786095619202, -0.1622653305530548, 0.1266818642616272, 0.19068244099617004, 0.6022018790245056, -0.23020237684249878, -0.41115468740463257, -1.3946837186813354, -1.2360866069793701, 0.6268138885498047, -0.9086915850639343, -0.19518591463565826, -2.0831587314605713, 0.04833998531103134, 1.5944809913635254, -0.7734907269477844, 0.1780313104391098, 0.7289543747901917, 0.7083325386047363, -0.8921840190887451, -0.41411179304122925, -1.2553861141204834, -0.30606895685195923, 0.37869420647621155, 0.4874509274959564, 0.9412367343902588, 1.1283589601516724, -0.1732318103313446, 0.7855703234672546, 1.3507976531982422, -0.5245537161827087, -0.7598548531532288, -1.9461442232131958, 0.15907317399978638, -1.090291142463684, -0.14483267068862915, 0.904548704624176, 0.40190616250038147, 0.7572178244590759, -0.9260380268096924, 1.360878825187683, -0.4002065360546112, 0.9731554388999939, 1.0955160856246948, 1.9269813299179077, -0.7201288342475891, 1.3612511157989502, 0.6002709269523621, 0.5913965702056885, 1.181817889213562, -1.8847113847732544, -0.9554584622383118, -1.1894042491912842, 1.3710756301879883, -0.5951661467552185, 1.2416529655456543, -0.4457241892814636, 0.503355860710144, 2.151362657546997, 1.7724982500076294, -0.21286630630493164, 0.13784147799015045, 1.1948906183242798, -0.45062729716300964, 1.1732197999954224, -1.11025071144104, -0.13116644322872162, 1.1487184762954712, 0.14201784133911133, -0.5800212025642395, -1.1375478506088257, 0.757466197013855, -0.5332444310188293, 0.3492579162120819, 0.38171425461769104, -1.5201271772384644, -0.43502986431121826, 0.304117351770401, 0.6297326683998108, 1.4113844633102417, 1.1865826845169067, 2.0579638481140137, -0.41077083349227905, 1.4299935102462769, 0.5374900698661804, -0.3698720633983612, -1.343467354774475, 0.12161314487457275, 1.4775665998458862, -0.40633344650268555, 1.0065103769302368, -0.5897600650787354, 2.4307804107666016, -1.5592705011367798, 0.6952264904975891, 0.2681567668914795, 0.6965725421905518, 1.1509383916854858, -1.1714489459991455, 1.1352779865264893, -0.859119713306427, 0.42045971751213074, 0.9937349557876587, 0.14313045144081116, 0.97596675157547, -0.3773703873157501, 1.9427074193954468, -0.10820458829402924, 0.6939107775688171, -0.7760346531867981, 0.9792103171348572, 0.17763757705688477, -0.26282259821891785, -0.36147838830947876, -0.35482358932495117, -0.26001015305519104, -0.47284626960754395, 0.5180053114891052, -0.8537233471870422, 0.030167749151587486, -0.18395155668258667, -1.2925621271133423, 0.0834641382098198, 0.09553693234920502, -1.9201664924621582, -2.5537209510803223, 0.34007224440574646, -1.8001800775527954, 1.0075688362121582, -0.2204902619123459, 1.8971080780029297, 0.6857621073722839, -1.2492822408676147, -0.6349102854728699, -0.26342228055000305, -0.27864009141921997, 2.3069040775299072, 0.6028688549995422, 0.7528314590454102, -1.8503155708312988, 2.2532248497009277, -0.5376229286193848, 0.6880108714103699, 1.3262838125228882, -0.5594189763069153, 0.0119674913585186, -0.32089710235595703, 2.4182872772216797, 1.799365758895874, 0.7368393540382385, -0.011743120849132538, -0.705757737159729, -0.3922814428806305, -0.29640382528305054, -1.9655444622039795, -1.427103042602539, -1.7678674459457397, 0.890933632850647, 0.15670141577720642, 0.8057278394699097, 0.027379034087061882, -0.34143489599227905, 0.7703980207443237, -0.7590201497077942, -0.5722630620002747, 0.4949421286582947, 0.01985621266067028, 0.454855740070343, -1.5893678665161133, 0.19304317235946655, 0.5694625377655029, 1.6919279098510742, 0.4886925518512726, -0.5155056715011597, -0.21531198918819427, -0.45688802003860474, 1.4051786661148071, -0.6906418204307556, 0.23788343369960785, 0.5474435687065125, 0.1803768128156662, 1.9047658443450928, -0.271379292011261, -0.3755047917366028, 1.3445929288864136, -0.23737448453903198, 1.24065363407135, 0.30254703760147095, 0.002907350193709135, -2.5241873264312744, 0.858273446559906, -0.0025693755596876144, 0.40381354093551636, 1.1119459867477417, -1.0065345764160156, -0.7715502977371216, 0.5104069709777832, 0.42136576771736145, 0.5917093753814697, 0.033046189695596695, 0.1893075555562973, -0.5975658893585205, -1.730432152748108, -1.1821647882461548, -0.2486226111650467, -0.1779830902814865, 2.20327091217041, -0.046835266053676605, -0.43730270862579346, -2.0031824111938477, -1.535170555114746, 0.10027306526899338, -0.906319797039032, 0.6219609379768372, 0.1498868465423584, 2.0548694133758545], [1.6438454389572144, -1.3498237133026123, -1.2463995218276978, -1.136243462562561, -0.41555890440940857, 1.2479878664016724, 0.9189817905426025, -0.3665240406990051, -1.0533115863800049, 0.3542550802230835, -0.936896562576294, -1.4362906217575073, -0.021185101941227913, -0.6166799664497375, 0.5962278246879578, 0.16516482830047607, 2.067007303237915, 0.2278631180524826, 0.1682881861925125, 0.8094592094421387, 0.017098959535360336, 0.8083482980728149, -0.5135807991027832, 1.2230550050735474, -1.008362889289856, 3.5976271629333496, -1.8828730583190918, 0.13654741644859314, 0.27739644050598145, -1.1395946741104126, 1.3261767625808716, 0.5056335926055908, -2.2182440757751465, 1.2146985530853271, -0.27187424898147583, -3.2376105785369873, 0.6231274604797363, -0.3678115904331207, 1.7625789642333984, 1.2760891914367676, -1.3049486875534058, 0.4782024919986725, 1.2052239179611206, 0.6992775797843933, -0.6645241975784302, 2.184889078140259, 0.4113963842391968, -0.12046005576848984, 0.4085204601287842, -0.1392032355070114, -0.06613247841596603, 1.9003021717071533, 0.589499294757843, 0.9316447377204895, 0.17741675674915314, 0.43734613060951233, 1.1586954593658447, -1.020156979560852, -1.0222104787826538, 2.273045539855957, 0.8340119123458862, 0.1371239423751831, -1.516839623451233, 0.8146964907646179, 0.8939124941825867, 1.26137375831604, -0.9351514577865601, 0.5080921053886414, -0.3318946659564972, 0.26202139258384705, -0.6255815029144287, 1.0421748161315918, -1.0319926738739014, 0.18818700313568115, -1.1466466188430786, -0.46289709210395813, 0.3213411867618561, 1.743865728378296, 2.065509557723999, -0.3616691529750824, -0.5814115405082703, 0.04876197129487991, -1.3629604578018188, 1.898855447769165, -0.1308593600988388, 0.5978395938873291, 0.7483488321304321, 1.479622721672058, -1.1261295080184937, -0.16340990364551544, 0.36605772376060486, 0.7520025968551636, -0.8354275226593018, 0.7091856002807617, -1.1060593128204346, -0.10906293988227844, -0.19696921110153198, -0.22704802453517914, 0.17963793873786926, 0.14258147776126862, 0.4736362099647522, -1.7303138971328735, 0.1974959820508957, -0.25886133313179016, -0.24577000737190247, -0.6100056171417236, -0.21116310358047485, -0.29441559314727783, 0.7871536612510681, 0.9747109413146973, -0.04130282253026962, 0.5882810950279236, 0.17648838460445404, 0.8157517313957214, -0.2547045052051544, -2.2371456623077393, 0.3383173644542694, 2.251706838607788, 2.415430784225464, -1.7339469194412231, -0.6481719613075256, 1.031843662261963, 2.0930864810943604, -2.7601382732391357, -0.4546423852443695, -0.644142746925354, 0.353092759847641, 0.3543734848499298, -0.7098851799964905, -0.10409371554851532, 2.4043030738830566, 0.4019254148006439, -0.5978200435638428, -0.008716604672372341, -0.01602700911462307, -1.1539103984832764, 0.049168966710567474, 0.3996277451515198, -1.2225561141967773, -1.1567397117614746, 0.6149104833602905, -0.1862659752368927, 0.03437773138284683, 0.2623632848262787, 0.4146903157234192, 2.2555980682373047, 0.38905319571495056, 0.7127669453620911, -0.9546058177947998, 1.0086512565612793, 0.5540465116500854, -0.1255294382572174, 0.439444899559021, 0.2173205316066742, 0.42123478651046753, -0.9138716459274292, 0.7053963541984558, 0.7370030283927917, -0.22336669266223907, 0.5171566605567932, 0.476902037858963, 1.0875020027160645, -1.955810308456421, -1.457432508468628, -1.4738675355911255, 0.04152857884764671, 0.2700875699520111, 0.04316829890012741, -1.8171336650848389, -0.9305593967437744, -0.6216700673103333, -1.288886308670044, -0.05464238300919533, -0.019807370379567146, 0.7189620733261108, -0.16005946695804596, 0.533943772315979, 0.4422428607940674, -0.04952920973300934, -0.46792975068092346, -0.65655916929245, 0.6694599390029907, -0.665222704410553, -1.2952673435211182, -0.3862921893596649, 0.5523008704185486, 0.04571118578314781, -0.30259108543395996, 1.569696307182312, 0.8102790713310242, -0.2984549403190613, -0.2963765859603882, 0.3908017575740814, 1.5496176481246948, -0.9878272414207458, -1.1080290079116821, -0.6115540862083435, -0.130219504237175, 1.1293094158172607, 2.559415340423584, -1.4099180698394775, 0.3759150505065918, 0.22737818956375122, -0.07740069180727005, 1.459276795387268, 0.26872318983078003, -0.9083500504493713, 0.7282248139381409, -1.1745638847351074, 1.6162317991256714, -0.24653837084770203, -0.6125773191452026, 0.228804349899292, 0.24278871715068817, -2.8496358394622803, 0.6303752064704895, -2.021169662475586, 1.0282580852508545, -0.21399177610874176, -1.781477451324463, 0.6996696591377258, -1.5293240547180176, 0.6989856958389282, 0.8595772385597229, -0.6689128875732422, -0.4984583854675293, -0.13679641485214233, -0.026317831128835678, -1.500483751296997, -1.0093590021133423, 0.13766035437583923, -1.1487090587615967, 0.4354797899723053, 0.4944492280483246, 1.5427196025848389, -1.4479048252105713, -1.4289811849594116, -1.1086043119430542, -0.44160884618759155, -0.5904673933982849, 0.48751696944236755, -2.226015567779541, 1.9153530597686768, -0.13238610327243805, 0.2700870633125305, 0.17405645549297333, -0.5573128461837769, -0.15643222630023956, 0.5796223282814026, -0.8216342329978943, -0.6197564005851746, 0.8227607607841492, 0.3604259192943573, 0.2661575675010681, 0.5590074062347412, -0.01006251946091652, -0.6890590190887451, 0.36565566062927246, 0.6567273736000061, 0.668721079826355, -1.3870905637741089, -0.5834615230560303, -0.410121887922287, 0.5204248428344727, 0.29131102561950684, -1.5503443479537964, 1.8713880777359009, -0.472245991230011, 1.3691718578338623, -0.42607420682907104, 1.4604872465133667, 0.11301036924123764, -1.394944667816162, 0.8753006458282471, 0.5810752511024475, -0.22154420614242554, 0.7584691643714905, 0.6486924290657043, -0.2008664458990097, -1.8799740076065063, -0.2826845049858093, -0.5074517726898193, 1.3167858123779297, 0.7747205495834351, -1.1490085124969482, 0.5487045645713806, -0.28016507625579834, -0.09445139020681381, 1.1110597848892212, 0.21947947144508362, 1.9722480773925781, -2.1295578479766846, 0.8907970786094666, 0.6127142906188965, 1.182283639907837, -0.4281602203845978, -0.44763845205307007, 0.5135796666145325, 1.099813461303711, -0.041464973241090775], [-0.06609981507062912, -1.073789358139038, -1.2874319553375244, -0.4814121127128601, 1.237308144569397, 0.5224705338478088, -1.5598124265670776, 0.03250347822904587, -0.9604394435882568, 1.357277750968933, -1.9759464263916016, -0.7229247093200684, -1.8091990947723389, 0.3343803584575653, -0.597032904624939, -0.7176017165184021, -1.5680643320083618, -0.2824404239654541, -0.07390256971120834, 1.2409273386001587, -0.8495199680328369, 0.2683617174625397, 1.2105768918991089, 1.521356225013733, -0.9336413145065308, 0.23686647415161133, -0.41168949007987976, 0.15984255075454712, 0.5699535012245178, -1.0168230533599854, 0.198523610830307, -1.9943904876708984, 0.6385531425476074, -0.3131190240383148, 1.001001000404358, -0.7486175298690796, 1.1996955871582031, 1.104093074798584, -0.5189806222915649, 0.9892348051071167, 0.42914319038391113, 0.8490285277366638, -0.1310049593448639, -1.1553319692611694, -2.0736072063446045, -0.013883719220757484, -0.5078294277191162, 1.5154651403427124, -1.943263053894043, -0.7707448601722717, 0.36181533336639404, -0.13184595108032227, -1.8070260286331177, 0.9167771339416504, -1.2354493141174316, 0.43895456194877625, -1.7614593505859375, -0.03795969858765602, -0.9994244575500488, 0.7545852661132812, -0.40740203857421875, 0.3413717746734619, -0.4449601471424103, -1.244879961013794, -2.0667779445648193, 0.8633618950843811, -1.3725577592849731, 0.44981926679611206, -0.11366105079650879, 0.30804353952407837, -0.05119071900844574, -0.45941781997680664, -0.9541552662849426, -0.7752168774604797, -1.1111410856246948, -0.7014225125312805, -1.5148684978485107, 0.1755722314119339, -0.4524505138397217, -0.4746633470058441, 1.5550457239151, -0.980673611164093, 1.2865996360778809, -1.0830498933792114, 0.7309119701385498, -0.8320729732513428, -1.6782333850860596, 0.41887909173965454, -0.35054993629455566, 0.1521286517381668, 0.8940354585647583, 1.2172431945800781, -0.11887682974338531, -0.4602106511592865, 0.8898965716362, 0.2852686047554016, -1.3981938362121582, 0.017166463658213615, 2.1133804321289062, 0.8055666089057922, -3.134704828262329, 0.7880265116691589, -0.9814741015434265, -0.33648091554641724, 0.41141802072525024, -0.12574653327465057, 0.23163199424743652, -0.029032906517386436, -0.3013235628604889, 0.6845760345458984, 0.24726340174674988, 1.2346763610839844, 1.8141820430755615, -0.560393750667572, -0.6169355511665344, 1.0207860469818115, -1.2170040607452393, -1.1009819507598877, -1.486953854560852, -1.3703893423080444, -0.09065863490104675, -0.7907182574272156, 1.7779839038848877, -0.6127560138702393, -0.7275042533874512, 0.2510989308357239, 1.9442042112350464, 0.7300643920898438, -0.20742888748645782, 0.08261378109455109, -0.936130166053772, 1.020872950553894, 0.2674235701560974, 0.290179967880249, 0.42675235867500305, 0.16004373133182526, 0.4393121600151062, 0.1772315353155136, -0.8427708745002747, 0.5467073321342468, -0.5090016722679138, 1.191996693611145, 0.7692763805389404, 0.37290164828300476, 0.9772974252700806, -0.19473083317279816, -0.7240249514579773, 1.0255132913589478, -1.2012978792190552, 1.6716876029968262, 0.3145962953567505, -0.8425703644752502, 0.4373003840446472, 0.5676035284996033, 0.26606062054634094, 1.721349835395813, 0.15042228996753693, 1.8327937126159668, -0.42996737360954285, -0.14336469769477844, 0.06640525162220001, -1.3025425672531128, 0.9713250994682312, 0.6475226879119873, 0.056985922157764435, 0.5982511639595032, 1.318996787071228, -0.7226150631904602, 1.4563623666763306, -0.0646095797419548, 1.0500502586364746, -0.6922513842582703, 0.545956015586853, -0.47680458426475525, -0.0676993653178215, -0.43345877528190613, 1.124709963798523, 0.01732461526989937, -0.6115179061889648, 1.1689488887786865, -1.0358572006225586, 1.2214982509613037, -0.11926137655973434, 0.171524316072464, 1.349721908569336, 0.3848530054092407, -0.0264559518545866, 0.7243876457214355, 2.3213436603546143, 0.48118147253990173, 0.4689420163631439, 1.7364319562911987, -0.7178144454956055, -1.5152974128723145, -0.08111422508955002, -0.9755488634109497, 0.053434502333402634, -1.1897510290145874, 0.5038064122200012, 1.7511866092681885, 1.410975694656372, 0.452048122882843, 0.8821587562561035, 2.4375953674316406, -0.4344193935394287, 0.32382315397262573, 1.2193183898925781, -1.5818933248519897, 0.6486572027206421, 0.07588188350200653, -0.1996680200099945, -1.1248496770858765, -0.29627200961112976, 1.0843138694763184, 0.06473395973443985, 1.4815014600753784, 0.6069533228874207, 0.20166173577308655, 2.066323757171631, 1.1151454448699951, -1.6058588027954102, -0.7950578927993774, 2.378753900527954, 1.684065818786621, -0.35324788093566895, 1.166096568107605, -1.041581630706787, -1.2398905754089355, 0.471612811088562, -0.9820830821990967, -0.0514824241399765, 0.8329529762268066, -0.53766268491745, -0.5510307550430298, 0.3022136986255646, -0.0385814793407917, -0.11371543258428574, -0.37157583236694336, -1.776822805404663, -1.1628543138504028, 1.6673730611801147, 0.29217270016670227, 0.8995325565338135, -0.09517007321119308, -1.6394600868225098, 1.3344426155090332, -0.5943633913993835, 0.4744965136051178, -0.06166478991508484, 0.36760759353637695, -1.2909681797027588, 0.3935316205024719, -1.263594627380371, 1.243808627128601, 0.324722558259964, 1.925364375114441, -0.7090471386909485, 0.8794689178466797, -1.804479956626892, -1.7276800870895386, -0.352055162191391, -0.13686025142669678, -0.6405653953552246, -0.17016619443893433, -0.52377849817276, 1.187416672706604, 0.17932623624801636, -0.4655499756336212, -0.10632573813199997, 1.1177012920379639, 0.4454943537712097, -0.11681918054819107, -0.08175897598266602, 0.15716871619224548, -0.5210568904876709, 0.7017974853515625, -0.9670760631561279, -1.3013702630996704, 1.071069359779358, -0.873898446559906, -0.6654435992240906, 0.45396777987480164, 0.340833842754364, 0.5896145105361938, -1.0449687242507935, -1.974075198173523, -1.1424025297164917, -1.4157863855361938, 0.43041881918907166, -0.0723864883184433, 0.7725793123245239, -1.7299062013626099, -2.0906577110290527, 1.2081503868103027, -0.5171858668327332, 0.3082839548587799, 0.7203854918479919, 2.676844835281372, -0.07243116199970245, 0.9317618012428284], [0.6007441878318787, -1.1379400491714478, -0.2905498743057251, -0.6158348321914673, 0.9969765543937683, 0.17434826493263245, -2.566027879714966, -0.08869944512844086, 0.13128696382045746, 1.9672867059707642, 0.4629616439342499, 0.4525551497936249, 0.8977513313293457, 0.27893924713134766, 1.0816564559936523, 0.37924090027809143, 0.19191007316112518, 0.6962203979492188, 0.20180653035640717, -2.7317512035369873, 0.17694562673568726, -1.8409456014633179, 1.0341215133666992, -0.0983910858631134, 2.2510182857513428, 0.5353039503097534, 2.006396532058716, 1.6747111082077026, -1.0580096244812012, 0.9763721823692322, 1.1371220350265503, -1.0036262273788452, 0.3708946108818054, -1.1079380512237549, -2.441208600997925, 1.362962245941162, -0.22045238316059113, -1.1247378587722778, 0.6228530406951904, -1.3670494556427002, 0.11537381261587143, -1.1190086603164673, -0.07228678464889526, 0.19673261046409607, 0.7214789986610413, -0.007678758818656206, -0.6846333742141724, -0.7041076421737671, 1.163830041885376, -0.5955665111541748, 0.2739482820034027, -0.7149205207824707, -0.16496844589710236, 1.5997248888015747, 0.8620661497116089, 0.49490079283714294, -0.9565805196762085, -1.0068646669387817, 1.3113675117492676, -0.10851623117923737, -0.7870728969573975, -1.4093817472457886, -1.0928925275802612, -0.29530370235443115, -0.43743255734443665, 0.016231665387749672, 0.1802445948123932, -0.8183981776237488, 0.4626108407974243, -2.054769992828369, -0.8413516879081726, -0.5598715543746948, 0.6129449009895325, 0.29415592551231384, 0.3390143811702728, -0.25190433859825134, -0.47479376196861267, 1.2000336647033691, 1.1932904720306396, -1.241302490234375, 1.5679073333740234, 0.8348118662834167, -0.0979260504245758, 0.11092016845941544, 1.8286255598068237, -1.1211340427398682, 0.015565313398838043, -0.2767241299152374, -1.6049656867980957, 1.9130100011825562, -1.5770169496536255, -1.8830084800720215, 0.05993710830807686, -0.22824066877365112, -0.14096009731292725, 0.35221007466316223, 0.4403418004512787, 0.8098063468933105, 1.052343726158142, -0.815180242061615, -2.235966920852661, -0.7278320789337158, -0.8322458863258362, -0.11901873350143433, -0.5142222046852112, 1.3905936479568481, 0.14380599558353424, -0.06579682976007462, -0.17329789698123932, -0.2495315819978714, 0.48452121019363403, -0.4638669192790985, 0.20573310554027557, 0.4170210659503937, 1.512350082397461, -0.7466281056404114, -0.31968218088150024, -0.7092797160148621, 0.9036266207695007, 0.7263129949569702, 0.1493789404630661, -0.3764800429344177, 0.6998551487922668, 0.511461615562439, -1.8613923788070679, -2.2926101684570312, 1.4045768976211548, -0.140158012509346, -1.0201233625411987, 0.9636809825897217, 0.4008055329322815, -0.7944813966751099, -0.4243887960910797, -0.5279120206832886, -0.7902594208717346, 1.8322148323059082, 0.3418978154659271, 0.2523626983165741, 2.516784191131592, 1.1298378705978394, 1.3952438831329346, 1.1231286525726318, -1.6646186113357544, -0.8928865790367126, 0.4992642402648926, 0.7863182425498962, 0.7419885396957397, -0.4078783094882965, -0.5182825922966003, 1.6248208284378052, -1.1994976997375488, -1.519295334815979, -0.5793785452842712, -1.0800970792770386, 0.4712166488170624, -0.6020912528038025, -2.317413330078125, -0.9621186852455139, 1.8659565448760986, 0.9669567346572876, -0.9519891142845154, -0.07526403665542603, 0.5642029643058777, -0.21999672055244446, 0.00045595664414577186, 0.14070366322994232, -1.2342722415924072, 0.5788801908493042, 1.415850281715393, 0.08206529170274734, -0.9319507479667664, -1.868420958518982, -1.4723539352416992, 1.467108964920044, 0.6326258182525635, 0.587650716304779, -2.556805372238159, -0.4310305118560791, -0.2559356689453125, -0.40688756108283997, -0.8048537969589233, -1.4060978889465332, -1.6084328889846802, -1.5125510692596436, -0.048052828758955, 0.18085996806621552, -0.184597909450531, 1.4033962488174438, 0.06407081335783005, -1.3879401683807373, -0.16497297585010529, -0.1260708123445511, -1.189582347869873, 3.074089765548706, -2.613154411315918, 0.3186848759651184, 1.46673583984375, -0.6064472198486328, -1.3123170137405396, -0.5014021992683411, 2.1681439876556396, 0.4968116879463196, -0.1676836907863617, 0.03733896091580391, 1.2247868776321411, -0.26261863112449646, -0.42182573676109314, 0.3215949237346649, -1.8511115312576294, -0.29379022121429443, 1.658537745475769, -2.622075319290161, 0.6640707850456238, 1.7279722690582275, 0.3229157626628876, 0.01649489253759384, 0.10512819141149521, 0.3485751748085022, 0.0081199174746871, -0.042709995061159134, 0.48319369554519653, -1.1743780374526978, 0.00512197520583868, -0.9974058866500854, -0.35488155484199524, -1.2359169721603394, 1.8193846940994263, -0.23519271612167358, 0.6279811263084412, 1.2558000087738037, -0.6471745371818542, -0.7070218920707703, 0.44629204273223877, -0.594504177570343, 1.3317151069641113, -0.9905166029930115, -0.33301490545272827, -1.4396469593048096, 0.7730380892753601, -2.396218776702881, -1.013663649559021, -1.1595255136489868, 1.1085679531097412, -0.4845830202102661, -0.7075744271278381, 2.1492271423339844, 1.5256901979446411, 2.1135666370391846, 0.18463288247585297, -0.8509535789489746, 1.189501166343689, -1.6145254373550415, 0.711494505405426, 0.1355140507221222, -0.6168126463890076, 1.5144881010055542, -0.20375144481658936, -0.09998069703578949, -0.272443026304245, -0.2381226122379303, 0.3244732916355133, 1.7828088998794556, 0.9372162222862244, -1.6384676694869995, -0.8245256543159485, 0.4641667604446411, -0.295259565114975, -0.6066304445266724, 0.47244688868522644, -1.6392606496810913, 0.19026951491832733, 0.5222735404968262, 1.5429010391235352, -0.5859284400939941, -1.7431938648223877, 0.38563236594200134, 1.4809749126434326, 0.4833739697933197, 0.4722844362258911, -1.1479475498199463, -1.3770203590393066, -1.2543044090270996, 1.043560266494751, 0.07174814492464066, -1.9389783143997192, -0.2033781111240387, -2.2445008754730225, -0.8427799940109253, -0.03272884339094162, 0.1503017246723175, 1.7311776876449585, -2.077373504638672, 1.1530091762542725, 2.110130548477173, 1.159877061843872, 0.5777836441993713, -1.7086464166641235, 0.7963035702705383, 0.19978396594524384, 1.2617226839065552], [1.3415230512619019, 1.5063790082931519, 0.3500901758670807, -1.0845718383789062, 0.8621188402175903, 0.24997563660144806, -0.09937377274036407, 0.6030263900756836, -1.3796736001968384, -0.4090600609779358, 1.172300934791565, 0.727618396282196, 0.3865327835083008, -0.49808642268180847, -0.05140316113829613, 0.4544517397880554, 1.176627516746521, 0.7821521162986755, 0.4715177118778229, 0.859492838382721, -0.20626763999462128, -0.2693816125392914, -0.4553421139717102, -0.4999716579914093, 0.1286766231060028, -0.9549149870872498, -0.576417088508606, -2.070204973220825, -0.0907004177570343, -0.9482412934303284, 1.6707179546356201, 0.2065085768699646, -1.390872836112976, 0.42779257893562317, 0.31139859557151794, 0.22736607491970062, -0.0025380339939147234, -0.279524564743042, -1.1653659343719482, 0.7717955112457275, 1.2012444734573364, -0.22455763816833496, -0.12495560944080353, -1.089489221572876, 0.26483532786369324, 1.3553990125656128, -1.341956615447998, 1.2846155166625977, 0.5649194717407227, 0.1738033890724182, 0.789872407913208, 0.1160641461610794, -1.5464760065078735, -0.034702103585004807, -0.7887467741966248, -0.56679368019104, -0.5443317890167236, 0.3907806873321533, -0.14265190064907074, -0.6131288409233093, 0.45583102107048035, 1.6648980379104614, -0.3726627230644226, -0.12504945695400238, 0.10964709520339966, -0.45743006467819214, 1.0277019739151, 1.1587166786193848, -0.07542729377746582, 0.730006754398346, -1.0567622184753418, -0.75223308801651, -0.9370068907737732, -0.9117144346237183, 0.71533203125, 0.33587706089019775, -0.6265350580215454, -0.14939114451408386, -0.78553706407547, -1.1778301000595093, 1.3185620307922363, -0.4043133556842804, 1.4367235898971558, 0.6108244061470032, 1.6800787448883057, 0.47514212131500244, -0.770247220993042, 0.13289637863636017, 0.8154664039611816, -0.020393306389451027, 0.6010270118713379, -0.36530184745788574, -0.7669652104377747, 1.8188074827194214, -3.0803940296173096, 1.5733755826950073, 0.1292503923177719, -0.5505682826042175, -0.03578580543398857, -0.03783879429101944, 0.08061476796865463, -0.16452884674072266, -0.5538893938064575, 0.553596019744873, 1.1423958539962769, -0.47064948081970215, -0.7220162749290466, 0.45305871963500977, -0.06007101386785507, -0.04777629300951958, 0.9063546657562256, 0.7363848090171814, -0.10124953091144562, 1.505955457687378, -1.8450714349746704, -0.2148829847574234, 0.11914759874343872, 0.8366853594779968, -0.19812385737895966, 0.3089413344860077, 0.6599465608596802, -0.42498183250427246, 0.7035810947418213, 0.4894740581512451, -1.2322505712509155, 1.4878289699554443, -2.1901097297668457, 0.5202534198760986, -0.5610536336898804, 0.33056360483169556, 1.0740728378295898, -0.5943825840950012, -1.289352297782898, -1.2971725463867188, 1.2521804571151733, 0.5743112564086914, -0.539409339427948, -0.08357570320367813, 0.38343796133995056, -0.5674777030944824, 1.0253245830535889, -0.20081180334091187, 0.17524538934230804, -0.6126254200935364, 0.515350878238678, 0.021606508642435074, -0.3461129069328308, 0.2707953155040741, -2.685389757156372, -0.46149781346321106, 1.082442283630371, -1.4545111656188965, -0.2668876349925995, 0.17477743327617645, 0.3770550489425659, 1.5342320203781128, 0.6693427562713623, 0.3773340582847595, 0.7216662168502808, 0.40316450595855713, -0.7100089192390442, 0.08168980479240417, -0.3050784766674042, -1.6656687259674072, 0.900320291519165, 0.9870395064353943, -0.960112452507019, -1.0011155605316162, 0.24432173371315002, 0.827225387096405, 2.0368752479553223, -1.550066590309143, -0.3215978145599365, 1.5268503427505493, -1.329740285873413, 1.4603866338729858, 0.8902270197868347, 0.3702298402786255, -0.4047253131866455, 0.6743720173835754, 0.6431136727333069, -0.1697508543729782, -1.088706135749817, -0.27836036682128906, -2.1129727363586426, 0.8147110342979431, 2.451225519180298, 0.26378491520881653, 0.6210028529167175, 1.0552399158477783, -1.6851823329925537, 0.0795551985502243, -1.259967565536499, -0.7513368725776672, 0.6285609006881714, -0.5566367506980896, 1.712912678718567, -0.13783101737499237, -0.986200749874115, 0.6593286991119385, -0.8508322238922119, -0.11893413960933685, -1.986188292503357, -1.1688169240951538, -0.5501397848129272, -0.2737162113189697, 0.4826275408267975, -0.3597736656665802, 1.5237787961959839, -0.6712594032287598, -0.6491367816925049, 1.4172394275665283, 0.8740968108177185, -1.2290014028549194, 0.985457181930542, 0.5630121827125549, -0.28916171193122864, 1.19557523727417, 1.7355793714523315, -0.8139360547065735, -1.5808771848678589, -0.6113555431365967, 0.8706908226013184, -0.6723311543464661, 1.0855568647384644, 1.4643079042434692, 1.1115355491638184, 0.1917165368795395, 0.020526817068457603, 0.1151847392320633, -1.4774407148361206, -0.6863607168197632, -0.0186359453946352, 0.7308617234230042, 0.23937192559242249, 0.09150601178407669, 0.5965132713317871, 0.6351620554924011, -2.7782552242279053, -0.6950356960296631, -0.5790528655052185, -1.8502296209335327, -0.751179575920105, 0.21709829568862915, 1.3877202272415161, 0.07658620178699493, 0.9643168449401855, 0.6070988178253174, -1.2595758438110352, 0.04516347125172615, 1.3698979616165161, 0.41735708713531494, -1.4724657535552979, -0.8473814725875854, 1.4310333728790283, 1.8633959293365479, 0.7783473134040833, 0.3151375949382782, -0.5901772975921631, 0.02194518782198429, 0.8981208205223083, 0.608116626739502, -0.29916638135910034, -0.6393269896507263, 0.9388852715492249, 1.4402694702148438, -0.8078733682632446, 0.20944903790950775, 1.316372036933899, 1.0734471082687378, 0.848503589630127, -1.0210399627685547, -1.2178494930267334, 0.7520039677619934, 2.2718729972839355, 0.12246061861515045, -1.072450876235962, -0.48123303055763245, -0.3991706371307373, -1.3261736631393433, -0.4308638870716095, 2.0451889038085938, 0.17812307178974152, -0.029532846063375473, -1.2078242301940918, -0.5634417533874512, -2.2657394409179688, 0.7311638593673706, 0.8187597393989563, 0.3018827438354492, -0.06089089810848236, -1.9761053323745728, 0.39200782775878906, 1.3582215309143066, 1.0496143102645874, 1.041507601737976, -0.8208106756210327, 0.8968287706375122, -0.4620106816291809, 0.832676887512207], [0.4158429503440857, 1.3455703258514404, -0.744327962398529, 0.46069467067718506, 0.7123185992240906, -0.06570079177618027, 0.2086397260427475, -0.016007879748940468, -0.5101020336151123, -0.3951737880706787, 0.7868515253067017, 0.1716141551733017, -0.7367492318153381, -1.2371045351028442, -0.9145548343658447, 0.6579709053039551, 0.4807209074497223, 1.3649368286132812, -0.08825498819351196, 0.7467937469482422, 0.25465118885040283, -0.4400651454925537, -0.32660412788391113, 1.1148130893707275, 1.405554175376892, -0.4718099534511566, 1.3398138284683228, 1.2762576341629028, 0.938997209072113, 0.0017979019321501255, -1.6318509578704834, -0.6481471657752991, 1.0929527282714844, -1.686513066291809, 0.49961555004119873, 0.22976772487163544, -0.438079833984375, -0.7781437635421753, 0.10516186058521271, 1.169823169708252, -1.1876072883605957, -0.6894878149032593, 1.9205107688903809, 0.5806902647018433, 0.9685627222061157, -0.6724069714546204, -1.123448371887207, -0.6423091888427734, -0.8020581007003784, -1.9358794689178467, -1.157158613204956, 0.2637016773223877, -0.3120167851448059, -1.0369422435760498, -0.14876340329647064, 1.1860557794570923, -2.1562352180480957, -0.25441786646842957, 0.09806357324123383, -0.1873328983783722, 0.20429085195064545, -0.5053525567054749, -0.325198769569397, 0.021763993427157402, -1.3231831789016724, 0.37022534012794495, -0.7399078607559204, -0.12948745489120483, -0.8008661270141602, -1.739192247390747, -0.08039414137601852, -0.7175898551940918, -1.2354909181594849, -0.5786964893341064, 1.3288366794586182, -0.3393446207046509, 0.21268148720264435, -1.1447489261627197, 0.2655212879180908, 1.0197826623916626, -1.5500617027282715, -2.0221574306488037, -0.16698004305362701, -0.7090487480163574, 0.9651100635528564, 0.8682929873466492, 1.801466941833496, 0.779580295085907, -0.3958776295185089, -0.45124804973602295, -0.9193577766418457, -0.011163460090756416, -0.2259008139371872, -0.26873040199279785, -0.06695663928985596, -0.35048872232437134, 0.72890305519104, -1.2363065481185913, 0.21987420320510864, 0.2570003271102905, 0.6022990345954895, 1.2881126403808594, -0.44042256474494934, 1.0476411581039429, -1.3647112846374512, 1.6634711027145386, 1.0642096996307373, -0.1066024899482727, 1.2543401718139648, 1.3012962341308594, -0.6793091893196106, 0.3625682294368744, 0.37864938378334045, -1.5172661542892456, 0.4155983030796051, 1.9602057933807373, 0.43252986669540405, -1.3193870782852173, -0.5407060980796814, -0.35130950808525085, -1.0592231750488281, 0.30841296911239624, 1.381219506263733, -0.6888127326965332, 0.17864438891410828, 0.08935714513063431, -0.5079411268234253, 0.2659894526004791, -1.3878285884857178, -1.5377005338668823, -1.8882156610488892, 1.8655997514724731, 1.1287187337875366, 0.034559279680252075, 1.003156304359436, 1.3570210933685303, 1.2924641370773315, -0.635532557964325, -0.4013146162033081, 0.22436921298503876, 1.8012105226516724, -0.010837728157639503, -1.8654290437698364, -0.07618114352226257, 1.5293389558792114, 1.4501397609710693, -0.17377375066280365, 0.1849701702594757, 0.913994550704956, 0.2929013669490814, -0.18652811646461487, -0.9128894805908203, 0.43009886145591736, -1.0501760244369507, -0.9514063596725464, -0.13192646205425262, -0.5415149331092834, 0.893392026424408, -0.45634976029396057, 1.9446377754211426, -0.5266477465629578, 0.7421817183494568, 0.14754067361354828, -2.7607710361480713, -0.37799131870269775, 0.7739327549934387, 1.9441206455230713, 0.056174200028181076, -1.7279140949249268, -0.5892802476882935, 0.9715794920921326, 1.0780596733093262, 0.8560981154441833, -0.8401890993118286, -0.1329500526189804, -0.22813557088375092, -0.7397212982177734, -0.997515082359314, -0.4192618727684021, -0.35940659046173096, -0.7166323661804199, -0.9525153040885925, -0.4122236371040344, 0.4358985722064972, -1.278647780418396, 0.4673371911048889, 0.7800003886222839, 0.03531120344996452, 0.6222217082977295, 0.2582987844944, 1.5892610549926758, 0.8156086802482605, 0.7794917225837708, -0.3859105408191681, -2.4151558876037598, -0.014961391687393188, -0.12300517410039902, 0.093928761780262, -0.6336625814437866, -0.20267245173454285, -0.23721083998680115, -0.34297630190849304, 0.127053365111351, -1.2038801908493042, -0.014437682926654816, -0.8670031428337097, 1.743735432624817, 1.3179278373718262, 1.7321809530258179, -1.0093276500701904, -0.02843780815601349, 0.06392691284418106, -1.037009596824646, 0.47859987616539, -0.5387821197509766, -0.29097703099250793, 0.13104693591594696, 0.08163255453109741, 0.06259368360042572, 0.9851217865943909, -0.20410744845867157, 1.0371456146240234, -1.5366679430007935, 0.034983210265636444, -0.17631296813488007, 0.9928601384162903, -0.44309911131858826, 0.2242748737335205, -0.23523889482021332, -1.3471579551696777, 0.17343901097774506, -0.18972550332546234, 0.3588324785232544, 0.27099040150642395, -0.9021713137626648, 2.0063345432281494, -0.06430398672819138, 0.10460296273231506, -0.2771093547344208, 0.5184482336044312, -1.18589448928833, -1.7605271339416504, 1.438524603843689, -0.36626294255256653, -1.5448236465454102, 1.0233476161956787, 1.1743804216384888, -0.8163676857948303, 2.6941003799438477, 0.8423104882240295, 0.1475394368171692, 0.7635181546211243, -0.35744503140449524, -0.888523280620575, 0.054934702813625336, -0.1697026938199997, 1.0843160152435303, -0.45942431688308716, -0.674771249294281, -1.7178630828857422, 0.7963493466377258, 0.15281203389167786, 0.19449077546596527, -2.071531295776367, 0.21786807477474213, 0.6722152829170227, 0.6739934682846069, -0.25625142455101013, -0.9526568055152893, 0.8698059916496277, -0.29598936438560486, -0.016199812293052673, -0.35881248116493225, -1.513441801071167, 0.884392499923706, 1.4143463373184204, 0.5720151662826538, 0.43501362204551697, -0.3179730474948883, -0.7023873925209045, 0.2644599676132202, -1.0293574333190918, -0.22142376005649567, 0.40789780020713806, 1.4741878509521484, 0.758994996547699, 1.4472827911376953, 1.7200894355773926, -0.5831360816955566, -1.3074475526809692, 0.5332486033439636, -0.6485838890075684, -0.7558892965316772, -0.2227618545293808, -0.9521602392196655, 0.7575416564941406, -1.039861798286438, 0.4801352322101593, -0.4328160881996155, 1.1545741558074951], [2.0185165405273438, 1.1947258710861206, -0.8414109349250793, 1.1225738525390625, 0.4237128496170044, -0.553178608417511, -0.2983204424381256, -0.41735032200813293, -0.15781649947166443, 0.6732460260391235, 0.8994892239570618, 0.8030998706817627, 2.4631669521331787, -1.3249709606170654, 0.6650708317756653, 0.2700903117656708, 1.4404897689819336, 0.2362411767244339, 0.86931973695755, 2.006267786026001, -0.8938472270965576, 1.2594666481018066, 0.5325446128845215, 0.10471128672361374, 0.04907676950097084, 2.1255643367767334, -0.8808310627937317, -0.7958093285560608, 1.463765263557434, -1.3256572484970093, 1.3954368829727173, 0.038161180913448334, -0.09885146468877792, 1.009235143661499, 0.3032974302768707, -0.08685307949781418, 0.3444437086582184, 0.4108411967754364, -0.6175358295440674, 1.3535233736038208, 0.1941496729850769, 1.1751251220703125, -0.2457958459854126, -0.8460887670516968, -1.0562976598739624, 0.8462361693382263, 0.25635480880737305, 0.4707467257976532, -0.0040351469069719315, -1.7845216989517212, -0.24792829155921936, -0.01716870628297329, -1.7057936191558838, -1.3479119539260864, -0.2121942788362503, -0.2044011950492859, -2.253530979156494, 0.8673239946365356, -0.3804578185081482, 0.9687731862068176, -1.3623738288879395, -0.6141382455825806, 1.2494395971298218, -0.6447049975395203, -1.435538649559021, -0.4901778995990753, 0.5665408372879028, -1.2849918603897095, -0.2292829304933548, -0.511117160320282, -0.8893781900405884, 0.2723526656627655, -1.219109058380127, 0.4229552745819092, 1.8597047328948975, -2.116939067840576, -2.7627756595611572, 1.3733372688293457, -0.4994193911552429, -0.9947384595870972, -1.01127028465271, 1.2356469631195068, 1.8517669439315796, 0.22481301426887512, -1.9798579216003418, -0.42757415771484375, -0.656430184841156, 0.04266766458749771, 1.5434651374816895, 0.4756944477558136, -0.8592344522476196, 0.02517346292734146, -0.49757739901542664, -2.3410065174102783, -0.15197066962718964, 0.22199003398418427, -1.0748614072799683, 0.9243971109390259, -1.978028416633606, 0.507407546043396, 1.2335236072540283, 0.3280623257160187, 0.20842215418815613, -0.8511349558830261, 0.1933961659669876, 0.7583078742027283, -0.12081486731767654, -0.6788223385810852, 1.9655511379241943, 0.9939242601394653, 0.22828024625778198, -0.5599626898765564, -1.4411462545394897, -0.8067882657051086, 0.4775291383266449, 0.5792431831359863, -1.1595232486724854, 0.6031395196914673, 0.6795420050621033, -0.21251153945922852, -0.17600014805793762, 2.034998893737793, -1.114225149154663, 0.10670726746320724, 1.0435631275177002, 0.7480818033218384, 0.7089745998382568, 0.5251132845878601, 0.2542462944984436, 0.14446982741355896, 0.021736841648817062, 0.40224868059158325, -1.2879878282546997, -1.548330307006836, -0.735619843006134, 0.2548363208770752, 0.9478468894958496, -1.9449349641799927, 0.999147355556488, 1.1492328643798828, 1.7465734481811523, 0.6907665133476257, 1.1349655389785767, -0.6734929084777832, -0.01043771579861641, 0.31621336936950684, -1.6888166666030884, -0.12079410254955292, 1.9674949645996094, -0.11096721887588501, -0.28176888823509216, 0.621942937374115, -1.951436996459961, -1.2043099403381348, -1.3866674900054932, 1.27874755859375, -0.8914120197296143, 2.2120769023895264, 2.5573511123657227, -0.18067878484725952, -0.27199608087539673, 0.3280054032802582, -0.6489509344100952, 1.0284204483032227, -0.8650244474411011, 0.4053793251514435, 0.5771907567977905, -1.58364737033844, -0.6209039092063904, 0.3522559106349945, 0.9164341688156128, 0.24939686059951782, 0.6521003842353821, 0.9084599018096924, -1.970680832862854, -0.3951808214187622, -1.049727201461792, 0.7633830308914185, 1.0688432455062866, 1.007064938545227, 0.449494332075119, 0.3073856234550476, -0.49052026867866516, 2.5185277462005615, -0.8732492923736572, -1.5839698314666748, 0.06816604733467102, 2.7841131687164307, -0.7990123629570007, -1.9391002655029297, -0.3442543148994446, 0.006713025271892548, -0.16603727638721466, 0.6829801797866821, -0.8651677966117859, -1.3937838077545166, 0.34302034974098206, 0.6033715605735779, 2.92490553855896, 1.7666614055633545, 0.8239985108375549, -1.8723430633544922, -0.08465917408466339, 1.417279601097107, 1.0322409868240356, 0.5852475166320801, 0.8706799745559692, 0.47178369760513306, -1.7601408958435059, -1.1129094362258911, 0.3102288544178009, 0.9653311371803284, 1.6229068040847778, 0.3038857877254486, 0.3742954730987549, -0.6529333591461182, 0.24060940742492676, 0.7620403170585632, -0.06195955350995064, -1.2237718105316162, 0.27447259426116943, -0.6380963325500488, -0.5748505592346191, -1.1593645811080933, -0.2556356191635132, 0.027378547936677933, -1.3605015277862549, 1.3502475023269653, -0.3591361343860626, -0.5570152997970581, 0.6207699775695801, 0.4399794340133667, -0.22242046892642975, 0.8123728036880493, -2.56197190284729, 1.40460205078125, -0.5588067173957825, 1.2732888460159302, -0.5184831023216248, 0.3356200158596039, 0.771937370300293, 0.056315790861845016, -0.797469973564148, 0.40155404806137085, 1.166318655014038, 1.4355580806732178, -0.9965522885322571, -0.566130518913269, -0.23227941989898682, 0.8994611501693726, 0.6671313047409058, 0.3950755000114441, 1.914433479309082, -0.26471683382987976, -1.4161336421966553, -0.23828335106372833, 0.8528090715408325, 0.7078030109405518, 0.0014413419412449002, -0.8752669095993042, -0.46992242336273193, -1.4084306955337524, -0.022840317338705063, -0.9200661778450012, -1.5162303447723389, -1.0424243211746216, 0.36844614148139954, 1.4080971479415894, -0.22848574817180634, 1.7916561365127563, -0.41517210006713867, 0.19754856824874878, -2.439896583557129, 1.3255517482757568, 0.7331998944282532, 0.6228475570678711, -1.5067559480667114, 0.8143393993377686, 0.5678668022155762, -1.1758452653884888, 0.2457907348871231, 0.5536670088768005, -0.40322399139404297, 1.0516879558563232, 2.2452239990234375, -0.7039453387260437, -0.08548085391521454, 1.1106476783752441, -0.13773122429847717, -0.252797931432724, 0.6288167834281921, -0.23526710271835327, -0.6114655137062073, 0.16182167828083038, 0.6837226748466492, 1.6341949701309204, 0.31333550810813904, 0.15836402773857117, -0.41363251209259033, 2.295027017593384], [-2.399325370788574, -0.49012675881385803, 2.046140432357788, -0.551021933555603, -1.2191823720932007, 0.8453917503356934, 0.5665245056152344, 0.07872315496206284, 0.610897421836853, -2.112488269805908, 0.9947173595428467, -1.581454873085022, 0.49296990036964417, -0.8583070635795593, -0.5022428035736084, -0.5361385941505432, -0.46765393018722534, -2.1461594104766846, -0.32731372117996216, 1.3687551021575928, -0.05841989442706108, 1.799798607826233, -1.1375411748886108, 1.4957181215286255, -0.2234574854373932, 0.9124969840049744, -1.0715938806533813, -2.324476480484009, -1.511529803276062, -1.9148838520050049, -2.035809278488159, 1.4590466022491455, 1.3760432004928589, 0.6441547274589539, 0.688395082950592, -0.6297838091850281, 0.6126372814178467, -0.8237225413322449, 0.6028160452842712, -1.6203595399856567, 0.8027864098548889, -0.06151210516691208, -1.287814736366272, -1.1145989894866943, 0.3066694736480713, 0.9883371591567993, 0.0031161715742200613, 0.7763985395431519, 0.4894046485424042, -0.024796931073069572, 2.304741382598877, -2.2053871154785156, 0.392333060503006, 0.8324527144432068, -0.700853168964386, -0.7774595618247986, 1.7797024250030518, 0.012212893925607204, 0.8776087760925293, 0.8249073028564453, 0.6501105427742004, -0.8057824373245239, -0.8540140390396118, 0.2682071924209595, -0.4984247386455536, 0.8885182738304138, 0.35227546095848083, 0.030893784016370773, -0.07361409068107605, -0.4561419188976288, -0.25734254717826843, -1.3778012990951538, -0.4339447319507599, 0.5788501501083374, -0.8133381605148315, 1.1348322629928589, -0.5149654746055603, -0.12743598222732544, 0.4865264296531677, 1.4497313499450684, 0.5304496884346008, 1.0279669761657715, 1.3944566249847412, 0.5799648761749268, -0.7370545268058777, -1.0512020587921143, -1.0162391662597656, 0.5478187203407288, -0.1455300897359848, 0.3471687138080597, -0.67115718126297, 0.0005387290730141103, 0.6978396773338318, -0.17141126096248627, 1.048669457435608, 1.2709887027740479, 0.5985967516899109, -0.11164796352386475, 0.37882208824157715, 0.4743824005126953, 0.6863601803779602, -0.1207943856716156, -0.7620009183883667, -1.6694397926330566, 0.4531191885471344, -1.4037091732025146, -1.5152136087417603, 0.13347098231315613, 0.4092717468738556, 0.9322214722633362, 0.881952702999115, -0.9779695272445679, 1.0137746334075928, -2.0511856079101562, -3.23018741607666, 2.545743227005005, 0.009505350142717361, 0.39425715804100037, 0.49771785736083984, -0.6391242742538452, -0.039620231837034225, 0.034797389060258865, -0.29503676295280457, -0.41894397139549255, 0.41907167434692383, -0.5234155058860779, 0.9469088315963745, -1.4354761838912964, -0.2905137836933136, -3.2033815383911133, 0.0720328539609909, -0.152299165725708, 0.8344609141349792, -0.5995168685913086, -3.157989740371704, 1.955554485321045, 0.6876983046531677, 1.1882396936416626, 1.1927738189697266, 1.0770087242126465, 1.0508753061294556, 0.16895349323749542, -0.06517399847507477, -0.2413877546787262, -0.8596949577331543, -0.7349399328231812, 1.1277459859848022, -0.9769502282142639, -2.69215989112854, -0.838020384311676, -3.07419490814209, -0.6726691126823425, 0.4834933578968048, 0.47635647654533386, -0.14091525971889496, 0.9765511155128479, -0.7156740427017212, 0.13990794122219086, -1.94118070602417, 0.5146611928939819, -0.7266722917556763, -0.4244830906391144, -2.267853021621704, 0.31882429122924805, 0.01578914001584053, 0.3530234098434448, -1.9662165641784668, -1.464091181755066, 1.2034817934036255, -0.4498876929283142, 0.7897542119026184, -0.051854535937309265, -0.008659985847771168, -0.14314259588718414, -1.6900612115859985, -1.5532617568969727, -0.7506810426712036, -0.6380453109741211, -2.8908889293670654, -0.27289143204689026, 1.655978798866272, -2.7640552520751953, 0.28175538778305054, 0.8619033694267273, -0.5360139012336731, 0.034031860530376434, -0.9267115592956543, 0.31349337100982666, 0.23968780040740967, 1.6471678018569946, 1.1251647472381592, 1.6391321420669556, 1.070816993713379, 0.3269634544849396, -2.559474229812622, -1.8979556560516357, 0.26869046688079834, 0.18476347625255585, -1.067458987236023, 0.8892987370491028, 0.9927600622177124, 1.1887931823730469, 0.9755638241767883, 0.17756301164627075, -1.3887927532196045, -0.6706864833831787, -0.418693870306015, 0.9630117416381836, 0.034654565155506134, 0.3732517659664154, -0.5611832141876221, -0.5454999208450317, -0.3671816885471344, 0.7412833571434021, 0.789351224899292, -0.7139158248901367, -0.5556648969650269, 1.1097972393035889, -0.032188281416893005, -0.013101414777338505, -0.8200442790985107, 0.6157709956169128, -0.03960580378770828, 0.4798702299594879, -0.957076907157898, -0.5125072598457336, 0.8508819341659546, -0.014871828258037567, 0.5175672173500061, -0.4368327558040619, -1.2903361320495605, 0.707663357257843, 0.4377935826778412, -0.2917018234729767, -0.16105180978775024, -0.863209068775177, -0.41946572065353394, -0.37528520822525024, 1.024895429611206, -0.45272311568260193, 2.1270389556884766, 1.8042025566101074, 1.0210251808166504, -0.4791499972343445, -0.9913089871406555, 1.1475862264633179, 0.3784234821796417, -0.21890783309936523, 1.5658174753189087, 0.7743121385574341, 0.28480470180511475, -1.874498963356018, -1.026403784751892, -0.5207434892654419, -0.12419051676988602, -0.15986737608909607, -1.2259559631347656, -1.087998867034912, 0.8873462080955505, 1.5994479656219482, 0.8383485078811646, 0.2564062774181366, 1.3243037462234497, 0.3627261519432068, -0.13374021649360657, 0.36278605461120605, 1.3027679920196533, -0.3867146372795105, -0.6958158612251282, -1.9652392864227295, 1.4949053525924683, -0.35054269433021545, 0.1756332963705063, -0.36714011430740356, 0.35663679242134094, -0.3085381090641022, 0.8982946276664734, 1.0495487451553345, 0.935706615447998, 0.879551351070404, -1.329681634902954, 0.500433087348938, -1.1938319206237793, 0.5347906351089478, -0.91518634557724, -1.959378957748413, -0.08943215757608414, 1.1075412034988403, -0.685210108757019, -2.0134246349334717, 0.7128574252128601, -1.2558108568191528, 2.103879451751709, 0.06534706056118011, -0.3074708580970764, 0.07682640105485916, -0.13772669434547424, 0.66605144739151, -0.019905760884284973, 1.6089719533920288], [-0.25922685861587524, -1.0607932806015015, -0.7541258931159973, 0.027317887172102928, -0.24252642691135406, -1.7661452293395996, -0.41652658581733704, -0.42265719175338745, -0.18690091371536255, -0.4029240906238556, -1.058483362197876, -1.7165701389312744, 1.8958103656768799, 2.490150213241577, -0.3144577443599701, -0.9622615575790405, 0.24157460033893585, -0.005605525337159634, -0.24571850895881653, -0.4015921950340271, -0.642597496509552, 1.0304595232009888, -0.24159248173236847, -0.9270201325416565, 0.6021785736083984, 1.0901575088500977, -1.7159488201141357, -0.10971902310848236, -0.11388324201107025, 1.9158543348312378, 0.05586693063378334, 0.5975486636161804, 0.2918296158313751, -0.03508322313427925, -0.06291236728429794, 0.38714197278022766, 0.4850941300392151, -3.473729133605957, -2.005951166152954, -0.5634517073631287, 1.3174500465393066, -0.29628583788871765, 0.7802143096923828, -0.2879212498664856, 0.757890522480011, -0.3223690688610077, -0.21943876147270203, 1.895962119102478, -0.5821361541748047, 0.3527967929840088, -2.0224995613098145, -1.8149328231811523, -0.30211347341537476, 1.0752222537994385, 0.5339159965515137, 1.0568000078201294, -0.7325363755226135, -0.24340499937534332, -1.478028416633606, 0.2741449177265167, -0.4655520021915436, -0.2902957499027252, 1.3570630550384521, 2.2749743461608887, -2.157487630844116, 0.04380864277482033, 1.1550359725952148, 0.7422493696212769, -2.3032703399658203, 0.6922303438186646, 0.05404629185795784, -0.8828890919685364, 0.4008275866508484, -0.2832513749599457, 0.4983191192150116, -1.713044285774231, -0.03342483937740326, -0.5157752633094788, -0.05419531092047691, 0.9483497142791748, 0.7381020188331604, 1.1450841426849365, 1.1440705060958862, -0.7930735945701599, 1.513192057609558, -0.2878323793411255, 1.305335521697998, -0.7867130637168884, -0.07246942818164825, 1.8191580772399902, -0.45757800340652466, -0.17711234092712402, -0.37797603011131287, -0.3788510262966156, -0.5966647267341614, -0.5640774965286255, -1.0160236358642578, 1.167262077331543, 0.021649830043315887, 1.306911826133728, -0.3041291832923889, 0.5109019875526428, 1.034940242767334, -0.6694441437721252, 0.8450379967689514, -0.6582860350608826, 0.6811307668685913, -0.4891085922718048, -1.3511039018630981, 0.6999519467353821, 0.08549370616674423, -0.1493096798658371, -0.5513684749603271, -0.2294989824295044, 0.16231510043144226, 0.42778050899505615, 0.7710563540458679, 0.8205774426460266, -1.5080370903015137, 1.909067153930664, -0.886651873588562, -0.8324501514434814, 0.1024758517742157, 1.9762080907821655, 1.0553843975067139, 1.3865880966186523, 0.5652645826339722, 1.6383720636367798, -0.7716873288154602, 0.5483102798461914, -1.203801155090332, -0.03427620604634285, 1.3399368524551392, 0.415788471698761, 1.0419923067092896, -0.3681492209434509, 1.131485104560852, -1.6336027383804321, 0.6640965938568115, -2.0974011421203613, 1.6795341968536377, -0.7773489356040955, -1.81387197971344, 0.37065431475639343, -0.11787550151348114, -0.5897124409675598, 2.079343318939209, -0.2522108852863312, -0.6639776825904846, -0.1860283464193344, -0.6672060489654541, -0.708665668964386, -0.2793470025062561, -0.24009566009044647, 0.7236448526382446, 1.0686815977096558, -2.1048176288604736, -0.2219328135251999, 0.11988678574562073, -0.5245415568351746, -0.6890102624893188, 0.427179753780365, 0.08436303585767746, -0.7115097641944885, -0.0042776502668857574, -0.13007085025310516, -0.24588824808597565, -0.20583949983119965, 0.08465759456157684, -0.8381792902946472, -0.08105870336294174, 0.6337534189224243, -0.07064864784479141, -0.24307136237621307, -0.36011964082717896, 2.0516257286071777, -0.4592004418373108, -0.5881430506706238, 0.17289616167545319, -0.6518633365631104, 1.915736436843872, -0.09187529981136322, -1.1713570356369019, -1.3752036094665527, 1.390547275543213, -1.0922369956970215, -1.2460399866104126, 1.4832618236541748, 0.9498562812805176, -0.7668247222900391, -2.1698129177093506, -0.5789584517478943, 1.3649953603744507, -1.2625070810317993, -0.2840089499950409, -0.05714170262217522, 0.14517824351787567, 0.013768262229859829, 0.7571954727172852, 0.584212064743042, 0.5811571478843689, 0.15612953901290894, 0.4763146638870239, 0.5926890969276428, -1.191602349281311, -0.602923572063446, -1.3435596227645874, -1.0021023750305176, -0.7840599417686462, 1.7768968343734741, -0.7397170066833496, 0.690146803855896, 1.0216182470321655, 1.6713924407958984, 0.3806692063808441, -0.7418293952941895, 1.4351099729537964, -0.779046893119812, -0.46583616733551025, -0.29256314039230347, -0.07450741529464722, 0.9920750260353088, -0.33580589294433594, -0.10351913422346115, 0.7315787672996521, 0.5455046892166138, -0.3443879187107086, 1.0713084936141968, 1.6065365076065063, -1.0750128030776978, 0.7513117790222168, 0.14870354533195496, 1.4499388933181763, 1.4107576608657837, 0.44172927737236023, -0.5324281454086304, 1.1159027814865112, 0.9202566742897034, -0.9630747437477112, -0.5544866323471069, -0.6266171336174011, 0.11798792332410812, -0.0615028478205204, -0.3401568830013275, 1.6866306066513062, 0.6372199654579163, -0.7106282711029053, 0.06321725994348526, 0.8520193696022034, -1.0154743194580078, -1.5226187705993652, 0.6736667156219482, 0.2420116811990738, 0.21750615537166595, 0.2675662636756897, -1.1485766172409058, 0.29679054021835327, -0.367617130279541, -2.196831464767456, -0.9350762367248535, 0.5467804670333862, 0.9814003109931946, 0.3308541774749756, -0.6520419120788574, 0.477044939994812, 2.2306976318359375, -0.10144507884979248, -0.00453891372308135, -0.2575607895851135, 1.0306745767593384, -0.606155514717102, -1.3954790830612183, 3.6327741146087646, 0.8287791609764099, 0.8006196022033691, -1.2747246026992798, -0.9640948176383972, 0.04743518680334091, -0.2535898983478546, 0.17683683335781097, -0.19190727174282074, 0.020435240119695663, 0.5953813791275024, 1.1933923959732056, -0.6296268701553345, 0.08546783775091171, 1.4151365756988525, -0.3732907176017761, 2.390260934829712, 0.7468811273574829, -0.5825935006141663, -0.015194414183497429, 0.5425944328308105, 1.6453518867492676, -1.6496189832687378, 0.3845333158969879, 1.2802835702896118, -0.8466744422912598, -0.08721433579921722, 0.5189304947853088], [-2.2379395961761475, 0.8353357911109924, -0.20849837362766266, 0.9504269957542419, -0.02881758101284504, 0.07513073831796646, 0.37634971737861633, -0.44648879766464233, -0.5580434799194336, 0.18744628131389618, -0.3279893100261688, -0.19755521416664124, 0.2134837657213211, -0.4346136450767517, 0.6102808713912964, 0.5892083048820496, 0.5494534373283386, 0.739284098148346, 0.168902188539505, -2.1403238773345947, -0.5717751979827881, -0.8900457620620728, -1.3158901929855347, -0.07145573943853378, 0.8871705532073975, -0.6482023000717163, -1.183979868888855, -0.5785566568374634, -0.2640325129032135, 1.092634677886963, 0.1784735769033432, 1.6571416854858398, 0.4210311770439148, -0.4166369140148163, -0.029519420117139816, -0.4791567027568817, 0.24471424520015717, 1.8342279195785522, 0.6770802736282349, 0.02276802808046341, -0.021183136850595474, 1.3492590188980103, -2.0786266326904297, -0.5393428802490234, -1.5825145244598389, 0.9509178400039673, 1.0046991109848022, 1.8290590047836304, 1.465874195098877, -0.500731348991394, -0.23387908935546875, -0.1920805275440216, 1.8723299503326416, -2.2789127826690674, 1.377254843711853, -0.8686202764511108, -0.5012613534927368, 0.17750230431556702, 0.3823990821838379, -0.8383284211158752, 0.5227113962173462, -0.08337902277708054, 1.4838262796401978, -0.3679791986942291, 0.8360369801521301, 1.090380072593689, -0.7242330312728882, 1.0816646814346313, 0.22758749127388, 1.3850667476654053, 0.617624819278717, 0.8790232539176941, -1.166336178779602, 0.20241688191890717, -0.9141194820404053, -0.9122939705848694, 0.5332318544387817, 0.2228306531906128, -2.4350075721740723, 0.09729675203561783, -0.5759646892547607, -1.6159969568252563, 0.5064627528190613, 1.7589998245239258, -2.064049005508423, -0.37412628531455994, 0.6607492566108704, -0.8520696759223938, 0.6882697343826294, -0.7708922624588013, 1.50298011302948, -0.06493319571018219, -0.34548863768577576, 0.7234674096107483, 0.4552370309829712, -0.6737209558486938, 0.4376939833164215, 0.8980055451393127, 0.24765072762966156, 0.8086416125297546, -0.7613713145256042, -0.7721092700958252, 0.6932446956634521, 0.677254855632782, 1.239469051361084, 1.4490313529968262, -1.172202467918396, -0.9418852925300598, -1.0773711204528809, -0.942273736000061, -2.123239040374756, -0.544791579246521, -0.8454772233963013, 1.7893459796905518, 1.788891315460205, 0.5603467226028442, -0.5694603323936462, 1.0219016075134277, 1.4816590547561646, -1.3291869163513184, -0.3929360806941986, 0.4591951072216034, -0.4017190933227539, -1.289246916770935, 0.16482840478420258, -0.5301415324211121, -0.2627383768558502, -1.383070707321167, -0.66730135679245, 0.9290855526924133, -3.116651773452759, 0.660780668258667, -3.3610451221466064, -1.3042174577713013, 0.052487462759017944, -0.30423709750175476, -1.2261987924575806, 0.5336385369300842, 0.34372538328170776, -1.1092065572738647, -1.3694506883621216, 0.17556704580783844, -0.11965996772050858, 1.3851666450500488, 2.0384414196014404, 1.2112438678741455, 1.4685920476913452, 0.2440468817949295, 0.0015851007774472237, -0.8791942000389099, 0.027471596375107765, 0.09925539791584015, 0.3248664140701294, -0.3342699706554413, -0.0453302301466465, -0.480770468711853, -0.03257551044225693, 0.7071974277496338, 0.3165576756000519, 1.1560392379760742, -0.10689732432365417, -0.1879039704799652, 0.10327713936567307, -1.7171361446380615, -0.8638578653335571, 0.9128419160842896, -0.3708939850330353, -1.032776117324829, 0.3766523599624634, 1.6636887788772583, -1.2261037826538086, -0.9721628427505493, 1.0331203937530518, -0.21729826927185059, 0.16085809469223022, 0.5553799867630005, -1.0946910381317139, -0.5373224020004272, -1.069265365600586, 0.07788971811532974, 1.8975881338119507, 0.19112712144851685, 0.7257586121559143, -0.35410764813423157, -0.23405036330223083, 0.7700874209403992, 0.3078506886959076, -0.7395519018173218, -1.4245564937591553, 0.684492290019989, 1.0617634057998657, 0.3212060332298279, -0.7230275273323059, 1.3571103811264038, 0.6352106332778931, -1.2992197275161743, 0.043586261570453644, 0.4078211784362793, 0.7215821146965027, 0.18768346309661865, 0.8266043066978455, -2.013946771621704, 0.03759185969829559, 0.5566663146018982, -0.2846078872680664, -0.5030367970466614, 0.8536899089813232, -1.3302797079086304, 0.517166256904602, 0.5855479836463928, -0.5934768915176392, 0.06846458464860916, 0.9182100892066956, 0.9360626339912415, -1.0529227256774902, 0.6323128938674927, -0.8233658671379089, 0.1159476563334465, 1.501478672027588, -0.25521090626716614, 0.6793174743652344, 0.3278079628944397, 0.85345458984375, 0.3202281594276428, 0.10759754478931427, -0.5141501426696777, 0.5258321166038513, -1.6780189275741577, -0.15408484637737274, -0.5718048810958862, 2.1932737827301025, -0.3236539959907532, -1.6658837795257568, 0.09792395681142807, -0.5186817049980164, -0.25909703969955444, 0.9890766143798828, -1.974746823310852, 0.02228478156030178, 0.1515214890241623, 0.5726340413093567, -1.424993872642517, -1.8758785724639893, -0.3462393283843994, 0.4017576575279236, -1.2633626461029053, -1.7566325664520264, -0.08656614273786545, 0.47063884139060974, 0.4698123037815094, -1.3860664367675781, 0.9344468712806702, 1.8267425298690796, -0.43405720591545105, 0.6984443664550781, 0.3218924105167389, -0.7713621854782104, 0.996916651725769, 0.9152049422264099, -0.13081032037734985, 0.3152490556240082, -0.9324192404747009, -0.11927314847707748, 0.7368962168693542, 0.2951135039329529, -0.8100714087486267, -0.027158847078680992, -1.202808141708374, 0.7478758096694946, 1.8789087533950806, -0.11509057134389877, -1.182307243347168, -0.23501771688461304, -1.057035207748413, -0.9462092518806458, 1.6458252668380737, 0.8253904581069946, -0.21187184751033783, -0.31337806582450867, -0.6426528692245483, 0.1497858464717865, -1.0152177810668945, 0.9980968236923218, -1.4777823686599731, -0.8154112696647644, -1.4316632747650146, 1.5262985229492188, 0.6645791530609131, -0.6456897258758545, -0.8765108585357666, -1.1877086162567139, -0.5206865072250366, 0.6046125292778015, 1.737808108329773, -0.043147310614585876, -1.145044207572937, -2.158418655395508, -1.6941419839859009, 0.9553207755088806, -0.8518452644348145], [1.0905708074569702, 0.17141467332839966, 0.4482506811618805, -1.6973212957382202, -1.7712788581848145, -1.2065867185592651, 0.585697591304779, -0.7137677073478699, 0.28088536858558655, -1.2872296571731567, 0.8132339119911194, -1.8426483869552612, 0.8898171186447144, -0.9817898869514465, 0.9278882145881653, 0.1334739476442337, -1.083753228187561, 0.4858143925666809, -0.7804295420646667, 0.24213473498821259, -1.7938296794891357, -0.39428409934043884, 0.36974987387657166, -0.19065351784229279, 0.8928250074386597, 0.4022389352321625, 0.5034298300743103, 0.3260582387447357, 0.09485310316085815, -0.46722254157066345, 0.3192202150821686, -0.7021079063415527, -2.329806089401245, -0.6649958491325378, 2.146778106689453, -0.012917682528495789, 0.4443005323410034, 0.8543439507484436, -3.4106760025024414, 0.31328463554382324, 1.3345849514007568, -0.9022256135940552, -0.3686445653438568, 0.0720943808555603, -0.46008458733558655, 0.04902759566903114, 0.38413605093955994, 0.9647742509841919, 0.0016015262808650732, 0.37174883484840393, -1.8296445608139038, -0.9371953010559082, 1.5058462619781494, 1.0473511219024658, 0.09645619988441467, -1.328277349472046, 0.6541493535041809, 0.2136874794960022, 0.533496081829071, -0.2240004688501358, 0.3237592875957489, -1.028927206993103, -0.3812500536441803, -0.33142998814582825, 1.0811837911605835, 0.2486606389284134, -1.3899649381637573, 0.009231975302100182, -0.4676026999950409, 1.6233443021774292, 0.44533485174179077, 1.2987918853759766, -0.298705518245697, 0.4391786456108093, 0.991174578666687, 0.48551619052886963, 0.3510752320289612, -1.808257818222046, -1.1886558532714844, -1.0829734802246094, 2.4476430416107178, 1.1986678838729858, 0.19547139108181, 0.2652299404144287, -1.6854791641235352, 0.858002245426178, 0.3829666078090668, -1.4330551624298096, 0.11302454769611359, -0.19362501800060272, 0.1356828808784485, -0.9380677938461304, 0.17939354479312897, 1.6231096982955933, -1.1906050443649292, -0.37045153975486755, 0.5271886587142944, -0.7403022646903992, -0.4336274266242981, -0.08909367024898529, 0.3275030255317688, -0.654621422290802, 0.6736893653869629, 2.2420754432678223, 0.5187336802482605, 0.5186493396759033, 0.2983896732330322, 0.5120012164115906, 0.2863427698612213, 1.7672345638275146, 0.15256938338279724, -0.7198846340179443, 0.7202810645103455, 1.5925958156585693, 2.400806188583374, -0.9685850143432617, -1.6483858823776245, -0.8030635118484497, 0.4353204369544983, 1.5607608556747437, 0.35210007429122925, 0.33121567964553833, -2.518599510192871, -0.2542128264904022, -0.9672212600708008, -1.0474635362625122, 0.16847065091133118, -2.400782823562622, 1.303008794784546, 1.8848732709884644, -0.7241836786270142, 1.2319852113723755, 0.3019927442073822, 1.858715295791626, 1.3976149559020996, 0.3167044520378113, 1.3527060747146606, 0.21724216639995575, 0.7806421518325806, -0.5631636381149292, 0.7769289016723633, 0.3806387484073639, 1.253463625907898, -1.877758264541626, -0.18376219272613525, 1.418273687362671, -0.6461305022239685, 0.7744066119194031, 0.719338059425354, -1.0237984657287598, -1.3898054361343384, -1.8917211294174194, -0.8255770206451416, -1.3360706567764282, 0.0834914818406105, 2.2451577186584473, 0.9483150839805603, -1.5946661233901978, 0.8184901475906372, 0.9330422282218933, -0.8961738348007202, 1.0253709554672241, -0.620391845703125, -0.8483744859695435, 0.2766304314136505, -0.7345360517501831, -0.7778694033622742, -3.298616886138916, -0.4080418646335602, -0.24728339910507202, -0.40927615761756897, 0.5121747255325317, -0.2555283010005951, 1.1730501651763916, -0.7337982654571533, -0.6292603015899658, 0.45608970522880554, 0.5044047236442566, -0.5197775363922119, 0.15489494800567627, -1.0134130716323853, 0.8575718402862549, -0.47149643301963806, 0.0613095760345459, -1.9339253902435303, -0.5458874106407166, 0.9351455569267273, 0.1961381435394287, -1.26170814037323, -0.6328542232513428, 0.2493409365415573, -0.4028852880001068, -0.24280382692813873, -0.291058212518692, 0.3921579122543335, -0.06011251360177994, -1.125218391418457, 0.8134348392486572, 0.016841206699609756, 0.5470111966133118, 1.1932027339935303, -1.2654409408569336, -0.6381517648696899, 1.693164348602295, 0.3690481185913086, 0.07240733504295349, -0.11691148579120636, 1.1620841026306152, -0.0018450943753123283, -1.5200968980789185, -2.6321189403533936, 0.49980324506759644, -0.4451431930065155, -1.588712453842163, 0.873455822467804, -0.5540647506713867, 0.059505194425582886, 0.7543848752975464, -0.6347559690475464, -0.5641025900840759, -0.4281744658946991, -1.2432425022125244, 0.12397385388612747, 0.24023745954036713, -1.4475332498550415, -0.5843176245689392, -0.17429931461811066, 0.5118037462234497, -1.3753986358642578, 0.7958177328109741, 0.6016963124275208, 1.4509429931640625, -0.22884105145931244, -0.2036130726337433, -2.632810592651367, 0.9453727602958679, -0.3822714686393738, 0.9752859473228455, -0.7999095916748047, 0.3180095851421356, -0.7488482594490051, 1.2916486263275146, 0.16382485628128052, 1.7935397624969482, -0.39658525586128235, -0.8625701069831848, -1.9356157779693604, -0.09468641132116318, 1.2011533975601196, 0.3482779562473297, 1.0352716445922852, 1.0097817182540894, 0.7047917246818542, -0.1392671763896942, -0.820637047290802, 0.17164845764636993, 0.43710026144981384, -0.7434282898902893, 0.41057053208351135, 0.22337256371974945, 2.1542088985443115, -0.3472581207752228, -1.5249841213226318, 0.14303497970104218, 1.4490655660629272, -0.9725915789604187, 1.0933878421783447, 0.42026230692863464, 0.4558880627155304, -1.3190276622772217, -0.20464709401130676, 0.8722324967384338, 0.9142388105392456, 1.4507160186767578, -1.3543529510498047, 0.8121210932731628, -0.7280192971229553, 1.889306664466858, 0.21064144372940063, -1.5392717123031616, 0.3867184817790985, -0.14418625831604004, 0.5652514696121216, -3.402177333831787, -1.2095348834991455, 0.9991114139556885, -0.0572989322245121, -0.026326976716518402, 0.31236040592193604, -0.7177237272262573, -0.07407738268375397, 0.29967162013053894, -0.9768999814987183, -0.29485541582107544, 0.5618990063667297, -0.23539899289608002, -0.41883018612861633, 0.5873051285743713, 0.4659780263900757, 0.9360607862472534], [-1.9104588031768799, -0.9399298429489136, 1.0962424278259277, -0.3071337044239044, -0.40986183285713196, 0.34868931770324707, 0.19286517798900604, -0.03212149068713188, -0.8981150984764099, 0.3059236705303192, 0.21565888822078705, 1.1244316101074219, -0.17802909016609192, -1.7628369331359863, 0.48109322786331177, -0.17519447207450867, -0.19467373192310333, -1.902540683746338, -1.1535552740097046, -1.1237050294876099, -0.6293855905532837, 0.09305060654878616, -0.7731948494911194, 0.4012328088283539, 0.43575912714004517, -0.02254284918308258, 0.6854934096336365, -0.3802346885204315, 0.25871017575263977, 0.8579073548316956, -0.027906250208616257, 0.18425166606903076, -2.5007965564727783, 1.4325153827667236, 1.259861946105957, -0.5961048007011414, 0.7240992784500122, 0.3392623960971832, 1.0928584337234497, 0.9849535226821899, 0.9995564222335815, 2.8115861415863037, -0.13586416840553284, -2.120112657546997, 0.2181432694196701, -0.5116311311721802, -0.6342026591300964, -0.4586631953716278, -1.1932038068771362, 1.3172821998596191, -0.9272273778915405, -2.5614588260650635, 0.4645758867263794, 2.867645263671875, 1.0169754028320312, -0.38294675946235657, -0.052036821842193604, 0.10480432957410812, 0.7404337525367737, -0.482562392950058, 0.21769092977046967, 0.618110716342926, -0.35753288865089417, 0.25765708088874817, 0.2779592275619507, 0.07653354108333588, -1.0409866571426392, 0.8336854577064514, 0.2061145007610321, -0.2524650990962982, -1.6284375190734863, 1.4478766918182373, -0.29861322045326233, 0.8957971334457397, -0.10935529321432114, 0.46647700667381287, -1.2853707075119019, -0.1112627387046814, -0.7097912430763245, 0.5189666152000427, 0.17139756679534912, 0.2512968182563782, -1.3603246212005615, 0.991726815700531, -0.6288317441940308, 0.6830725073814392, -0.0013842439511790872, -0.1363697350025177, 0.3746132254600525, -1.6241282224655151, -1.3260948657989502, 0.4052521586418152, 1.1365503072738647, 0.5214042067527771, 0.09604667872190475, -1.0247637033462524, -0.7927013039588928, -1.0211756229400635, 0.4437493681907654, 0.17765206098556519, 0.02333230711519718, -1.0002861022949219, 1.2890868186950684, 0.999109148979187, -0.5801137089729309, -1.7773137092590332, 0.028250720351934433, -0.06211860105395317, 1.4812179803848267, 1.0851606130599976, -0.6948665976524353, 0.17442962527275085, -0.32565411925315857, 0.37302300333976746, 0.8175537586212158, -1.49614417552948, 0.4336353540420532, -0.12675823271274567, -0.028333835303783417, 0.19319671392440796, 0.20031456649303436, -0.6179026365280151, -0.8688963055610657, -1.1050877571105957, -1.4189887046813965, -0.33530178666114807, 1.7900899648666382, 0.9367110133171082, -0.8459281921386719, -0.779984176158905, 0.9528278708457947, -0.5513456463813782, 0.19840075075626373, -0.9858258962631226, 0.07513822615146637, -0.19693778455257416, 0.9200603365898132, -1.3794187307357788, -0.7497742772102356, -0.5744481086730957, -0.11311553418636322, 1.0806210041046143, 0.5314938426017761, 0.24337439239025116, -0.46264463663101196, -0.47895124554634094, -0.3145931363105774, -1.601987361907959, 0.3480559289455414, -0.6159890294075012, -1.1374125480651855, 1.919469952583313, -0.8029041290283203, 0.8259097337722778, 0.3635958731174469, -2.2948081493377686, 1.1016496419906616, 0.19893664121627808, -0.7896551489830017, -0.37635597586631775, -0.6202170848846436, 0.8375349640846252, 0.13490335643291473, 0.5264343619346619, 1.0733586549758911, 1.5194953680038452, -1.0363565683364868, 0.22764723002910614, 0.4170723557472229, -0.4070899784564972, 0.2134627252817154, 1.2582154273986816, -1.7860465049743652, 1.612943410873413, 0.17643097043037415, -0.2671789526939392, 0.05957116186618805, -0.45894816517829895, -2.0720815658569336, -0.38527730107307434, -0.6513029932975769, -1.6175568103790283, -0.5464766621589661, 1.1440186500549316, 0.3923768103122711, -0.13404521346092224, 0.09677746146917343, 0.09468545019626617, 0.4526568055152893, -0.14517442882061005, 0.8615527749061584, -1.5151331424713135, 1.1930359601974487, 0.862962543964386, 0.19171501696109772, 0.7773522734642029, -0.3565337657928467, -0.4156486690044403, -0.9957486391067505, -1.043054223060608, 0.7663429379463196, -1.2186352014541626, 0.017244622111320496, 0.45977532863616943, 0.5665097832679749, 1.5218696594238281, -1.5761598348617554, 0.727337121963501, -0.5639799237251282, 0.23590153455734253, 1.421440601348877, -0.6809225082397461, -0.7928917407989502, 0.2832799255847931, 1.33660089969635, -1.3706493377685547, 0.7844898104667664, -0.09589605778455734, 0.16104033589363098, 0.531541645526886, -1.2462806701660156, 1.3725943565368652, 0.08761152625083923, -1.8826112747192383, 0.7610229849815369, -0.7613727450370789, -0.5479995608329773, -1.0984259843826294, -2.550325393676758, 1.6452397108078003, -0.5806463360786438, 1.3712129592895508, 0.0453585721552372, 0.4758179187774658, -0.2339611053466797, -0.44455817341804504, -1.6653121709823608, -0.8889954686164856, 0.1588003784418106, 0.4124397337436676, -0.8478481769561768, -0.2837880253791809, 1.141853928565979, 0.24883028864860535, -0.7579100131988525, -0.49148616194725037, 0.07412795722484589, -0.831296980381012, 0.004187510348856449, -0.5657652616500854, -0.11861499398946762, 0.09729072451591492, 0.04588083550333977, -1.2142263650894165, 0.0819951519370079, -0.8792898058891296, -0.1264120489358902, 1.6592367887496948, 0.4253898561000824, -2.3754143714904785, 0.6168767213821411, -0.06918289512395859, 0.6670523285865784, -1.5954620838165283, -0.7680441737174988, 0.1420864462852478, -0.445063054561615, -1.6852689981460571, -0.19871044158935547, -1.511922001838684, -2.0449821949005127, 0.016198603436350822, 1.281808853149414, -1.1165518760681152, -0.035368386656045914, 0.6446797847747803, -1.255009651184082, -0.403959184885025, 0.9296552538871765, -0.002053383504971862, 0.7816051840782166, -0.3322012424468994, 1.0702550411224365, -0.059925973415374756, 0.5556168556213379, -0.5989044904708862, -0.31293097138404846, 0.9813573360443115, -0.37477314472198486, -0.4540507197380066, -0.9288439750671387, 0.36051055788993835, -1.049990177154541, 0.39880603551864624, 0.6333755254745483, -1.7592322826385498, 0.07626201212406158, -1.1360290050506592, 0.7830436825752258, 1.6452475786209106], [1.8997822999954224, -0.4710770845413208, -0.26412302255630493, -0.1430509388446808, 0.9926803112030029, -0.4379838705062866, 1.7637969255447388, 0.01535376999527216, 2.2844855785369873, -0.34007108211517334, 0.6875183582305908, 0.9225345253944397, -0.5235239267349243, 0.6241957545280457, 1.1475869417190552, 0.6764468550682068, 1.9096245765686035, 0.4040776789188385, 0.10406067222356796, 1.8391778469085693, 1.6525156497955322, -0.9590578079223633, 1.261145830154419, -0.744067907333374, 1.0444658994674683, -0.44679325819015503, -0.7085585594177246, -2.282053232192993, 1.354034662246704, 0.6908501982688904, -2.0215518474578857, -0.6203879714012146, 1.0981636047363281, -1.188506841659546, -1.0714339017868042, 0.3733671009540558, 0.15433825552463531, -0.12868690490722656, 1.551421880722046, 0.21578431129455566, 0.390596866607666, 1.2519617080688477, 0.30909520387649536, 0.5123881697654724, -0.6066571474075317, -0.6370818614959717, 0.21527978777885437, 0.027543043717741966, 2.057126760482788, 0.8924517035484314, 0.6389753818511963, -0.323716938495636, 0.3738352358341217, 0.3719825744628906, -0.925148069858551, 0.4246528744697571, 1.7821440696716309, 0.8520739078521729, -2.503031015396118, -0.752312958240509, -0.014315273612737656, -0.364618718624115, -0.9096761345863342, -0.6353815197944641, -1.1678014993667603, 0.2579938769340515, -0.06309086829423904, 1.5563446283340454, -0.021173052489757538, 0.8740416765213013, -0.5855298638343811, 0.5530685782432556, 0.34086549282073975, -0.5424426198005676, -1.421944260597229, -1.402183175086975, -1.3397520780563354, 0.5648732781410217, 0.18804243206977844, -1.8682349920272827, -1.8329308032989502, 0.2744530141353607, 0.06693960726261139, 0.15060356259346008, -0.8696812391281128, -0.11781488358974457, -1.3034045696258545, -0.9586405158042908, 0.0012642682995647192, -0.1368788480758667, 0.9146520495414734, -0.5677000284194946, 1.279417872428894, 0.2453271597623825, 0.9277480244636536, 0.8009470701217651, -0.4517075717449188, -1.6946630477905273, 0.5783482789993286, -1.0547678470611572, -0.6797627210617065, -0.8971729874610901, 0.26017385721206665, -1.9783872365951538, 3.0217807292938232, -2.164694309234619, -0.36770349740982056, 0.17553196847438812, -0.3484549820423126, -1.273369312286377, -1.5754343271255493, 0.6809061169624329, -0.33141273260116577, 1.093239188194275, -1.168198585510254, 0.49209028482437134, 0.07950229197740555, 0.21017953753471375, 2.0975282192230225, -0.5946354866027832, -0.09122606366872787, -1.25497567653656, -1.113713026046753, -0.271998792886734, 1.7379074096679688, 0.12175443023443222, 0.8118782043457031, -1.1864286661148071, -0.8181986212730408, -1.3156312704086304, -0.13383257389068604, -1.1298786401748657, 0.24986201524734497, 1.7022072076797485, 0.12980565428733826, 1.6008305549621582, 0.40298062562942505, -0.9178543090820312, 0.44206953048706055, 1.6513060331344604, -1.1947509050369263, -0.8193214535713196, -0.3555779755115509, -0.10624133795499802, -0.2161414623260498, 0.9122492671012878, 1.5216137170791626, -0.7698550224304199, 0.8603277802467346, -0.9144518971443176, -1.2554960250854492, 0.08313266932964325, -0.21426595747470856, 0.6585907340049744, 0.5934650897979736, -0.31571266055107117, 0.6123284697532654, 0.29843977093696594, -0.048763372004032135, 0.4585229754447937, 0.6332690119743347, -0.2859782576560974, -0.1520412415266037, 1.099078893661499, 0.6527618169784546, -0.6744599938392639, 0.018838217481970787, -1.3776217699050903, -1.447797179222107, -0.2749280333518982, -0.5439059138298035, 0.0016988550778478384, -0.8144923448562622, -0.1392238587141037, 0.9623252153396606, 1.5532002449035645, -1.0888640880584717, 0.4837537705898285, 1.863505244255066, 0.48287516832351685, -1.7969746589660645, -0.31914108991622925, 1.0352262258529663, 0.09287422895431519, 0.23856526613235474, 0.36671656370162964, -1.2971627712249756, 0.5197838544845581, -0.37545353174209595, 0.1676120012998581, -2.4458935260772705, 1.0750064849853516, 1.0689940452575684, 0.4133750796318054, -0.21530388295650482, 1.3529682159423828, 2.0068187713623047, -0.8163345456123352, -0.8334342241287231, -1.891160488128662, 0.9725601077079773, -0.6822478771209717, 1.4800429344177246, 0.3102441132068634, -0.5903177261352539, -0.19844400882720947, -1.8167635202407837, -0.8272650837898254, -0.6437879800796509, 0.9597446918487549, 1.4355708360671997, -0.8996702432632446, 0.41438668966293335, 0.7921209335327148, -1.1497113704681396, -0.306748628616333, -1.7344838852295652e-05, -0.9883497357368469, 0.3429783284664154, 1.8603733777999878, -1.215754508972168, -0.8690665364265442, 0.7347750663757324, 0.2024240642786026, -0.4481674134731293, -1.0571621656417847, 0.004912998527288437, 0.8542304039001465, 1.4961566925048828, 0.07363438606262207, -0.07872692495584488, 1.0134791135787964, -0.3725578784942627, 0.615941047668457, 1.0029109716415405, -0.1997527927160263, 1.1609641313552856, 0.4321262538433075, -0.9263365268707275, 0.49897027015686035, 1.16677725315094, 1.452595829963684, -0.5689864158630371, -1.3082807064056396, -0.7449710965156555, 1.4179189205169678, -0.8663988709449768, 1.1749749183654785, -1.2773854732513428, -1.8334228992462158, 0.2802753746509552, -0.3071635067462921, 1.7068151235580444, 0.07204198837280273, 0.24082785844802856, 0.03888360410928726, -1.6914688348770142, 0.08433907479047775, 0.4177705943584442, -1.1079686880111694, -0.1924121379852295, 0.6318785548210144, -0.11740128695964813, -0.739605724811554, -1.7452080249786377, 0.8678416609764099, -0.2364043891429901, 0.04141204059123993, -0.9933421015739441, -0.6657462120056152, 1.227400779724121, 0.3664347529411316, -1.5034992694854736, -0.11584392935037613, 0.31383469700813293, -0.10258355736732483, -0.7952444553375244, 0.6611402630805969, 0.4042963981628418, -0.07987245172262192, -0.6521955728530884, -1.5824010372161865, 0.7349957227706909, 0.29063674807548523, -0.09317748993635178, 0.730827808380127, 1.1265759468078613, 0.3392731845378876, 0.7219124436378479, -0.4871767461299896, 0.22523751854896545, -0.2211347222328186, -0.48481860756874084, -0.08867058157920837, -1.2867835760116577, 0.37664979696273804, 0.12156078964471817, 0.633579671382904, -1.3756893873214722, -0.530925452709198], [1.7603654861450195, -0.528680145740509, -2.3702924251556396, -1.1218510866165161, 1.5285738706588745, 0.5773807168006897, -0.49297046661376953, 0.11259404569864273, 0.17678014934062958, 0.20588771998882294, 2.0774521827697754, -0.25332802534103394, -0.22909152507781982, 1.7217626571655273, 0.3861025869846344, -1.550593614578247, 0.7491310834884644, 0.008307154290378094, 2.875502586364746, -0.014427323825657368, -0.3995034992694855, -0.3180404305458069, -1.6622925996780396, -0.3455300033092499, -1.6524012088775635, 0.12705691158771515, -0.4126662313938141, 0.6737976670265198, -0.3561049699783325, -2.334986925125122, 0.18686865270137787, 0.16430611908435822, 1.1309006214141846, -0.7427741289138794, 0.6737690567970276, 0.7403594255447388, -3.117095947265625, -0.8140720129013062, -1.9660999774932861, 0.11638738960027695, 1.0287166833877563, 0.9445532560348511, 0.02477174438536167, 1.3061153888702393, -1.2563170194625854, 0.42137011885643005, -0.6361031532287598, -0.1973101943731308, -0.3176431655883789, 1.243967056274414, 1.1122868061065674, 0.38128653168678284, -0.4142290949821472, -1.9349533319473267, 0.4100453555583954, 0.9848175048828125, -0.16027489304542542, 0.8455408215522766, -0.42073357105255127, 0.03601525351405144, -0.09773551672697067, 3.723414421081543, 1.229870319366455, 1.491725206375122, 1.199761152267456, -0.34908604621887207, 0.4230428636074066, -0.3441014289855957, 0.950839638710022, -0.23968148231506348, -0.478048712015152, -1.7648544311523438, -1.3881734609603882, -0.05951036512851715, -0.05149075388908386, 0.676569402217865, -0.3171157240867615, -1.1217114925384521, 0.36259225010871887, -0.5886554718017578, -0.09209004044532776, -1.0306414365768433, -1.413329839706421, -0.7820738554000854, -1.4405980110168457, 2.0000417232513428, 0.13133825361728668, -0.2136581987142563, -1.4808411598205566, 0.3785005211830139, 0.8122828006744385, 1.8758562803268433, 0.17036013305187225, -0.4536873996257782, -0.034382812678813934, 1.915780782699585, 0.47241756319999695, 0.6249505281448364, 0.8126251697540283, 0.5258374810218811, 0.07011284679174423, -1.5056954622268677, -1.0457078218460083, 1.1919230222702026, 0.6886366605758667, 1.3491294384002686, 1.2681679725646973, 0.0323476567864418, -1.1915152072906494, -0.3442712724208832, 0.5117409825325012, -0.7032682299613953, 0.26308467984199524, 0.536935567855835, -1.8356152772903442, -0.5423159003257751, 0.1042865440249443, -0.3406084179878235, 0.28757572174072266, -0.8839600682258606, -0.17191274464130402, 0.5909498333930969, -0.3096873164176941, -1.6826913356781006, -0.4350297749042511, -0.3743349611759186, -0.4206506907939911, -0.4118023216724396, -0.31471818685531616, -0.6336467266082764, 0.22386080026626587, -0.12384264916181564, 0.7026314735412598, -1.3738651275634766, 1.3961873054504395, 0.14797012507915497, 1.5235718488693237, -0.7326627969741821, 0.2561696469783783, -0.36339694261550903, 0.4544105529785156, 0.5499762892723083, -0.6862262487411499, 0.07153403759002686, -1.6601756811141968, -1.2386975288391113, -0.047940995544195175, 0.6452057361602783, 0.3307569622993469, -0.22164645791053772, 1.3646749258041382, 0.4966033399105072, 0.9049232602119446, -0.6789127588272095, 0.8402764797210693, 0.013171078637242317, 0.45207902789115906, -0.1866137534379959, -0.7000409364700317, 0.9528955817222595, -1.084898829460144, -0.452946275472641, -0.46298354864120483, -1.4126272201538086, -0.2897895574569702, 0.5518167614936829, 0.7940948009490967, 0.46695154905319214, -2.4151012897491455, -1.3460744619369507, 1.0151070356369019, 0.31723257899284363, 0.37988677620887756, -1.1758431196212769, 0.3932163417339325, 2.637336254119873, 0.0891551747918129, 1.0899364948272705, -1.12296724319458, 0.5148561000823975, -0.866635262966156, 0.7339877486228943, -1.125410795211792, 0.21423618495464325, -0.6715878248214722, 1.109222173690796, 1.3776824474334717, -1.2242228984832764, -0.049588944762945175, -0.2938652038574219, 0.20400388538837433, 3.152982234954834, -0.7721383571624756, 0.14530320465564728, -1.388410210609436, 0.02194138430058956, -0.05087653547525406, -0.1870313137769699, 0.6527760028839111, 0.7876325249671936, -0.5723661184310913, 0.0723370760679245, 0.29608744382858276, -0.65311199426651, 0.773189902305603, -2.097567081451416, 0.2571665346622467, -0.16111533343791962, 0.08681333810091019, -1.1599863767623901, 0.6820626258850098, 0.015053428709506989, -0.4777945280075073, -0.8006705641746521, -2.2259387969970703, 0.023413797840476036, -2.0767810344696045, -1.2909126281738281, 0.3735582232475281, 0.9322343468666077, -0.01965038664638996, 0.9482849836349487, 0.1908450424671173, 0.20228196680545807, -1.0143698453903198, 0.006372606847435236, 0.8542177677154541, 0.42908817529678345, -0.6961300373077393, 0.008669615723192692, -0.9537178874015808, 0.43220266699790955, -0.655200719833374, -1.052842140197754, -0.6844517588615417, 0.0404219813644886, 0.505259096622467, -1.6284986734390259, -0.2722080647945404, 0.748246967792511, -0.8330541253089905, 0.9882472157478333, -2.3595807552337646, 0.8510625958442688, 0.11222679167985916, 0.41612228751182556, -0.03220705687999725, 0.2533048987388611, 1.480592966079712, 0.08951500058174133, -0.9531396627426147, -0.014287005178630352, 0.22920681536197662, 2.2256858348846436, 1.6645160913467407, -0.056541912257671356, 1.455496072769165, 0.7243381142616272, -0.32316187024116516, -0.9731019735336304, 0.5502764582633972, -2.7193939685821533, -0.3282991349697113, -0.3344508409500122, -2.1856069564819336, -0.29043737053871155, 1.4152429103851318, -0.47451072931289673, 2.4531712532043457, -1.1517302989959717, -0.0020642902236431837, 0.614234447479248, 0.7388310432434082, -0.9532628059387207, -0.9397693872451782, 0.7205203771591187, -0.2576444447040558, 1.1876262426376343, 0.8372301459312439, 0.11835784465074539, -1.3662644624710083, 1.7726852893829346, -1.4438892602920532, -1.3266137838363647, 0.24892981350421906, -1.0788304805755615, 0.9310541749000549, -0.12573176622390747, 1.0779070854187012, -0.18321844935417175, 0.9961342215538025, 0.6285695433616638, -1.098788857460022, -0.5213592052459717, -1.0745846033096313, 1.1338551044464111, 0.8060817122459412, 0.5666171312332153, 0.046442240476608276, 1.0045243501663208], [-1.039467453956604, -0.13366633653640747, -0.6241010427474976, 0.04780791327357292, 0.4716227650642395, -0.9365105032920837, -0.7096441388130188, 0.6493596434593201, -0.15621432662010193, 0.1073824092745781, -0.8357229828834534, 0.16451692581176758, 0.3862706124782562, -0.6550098061561584, -0.2724214494228363, 1.926756501197815, 0.058379095047712326, -1.427777886390686, 0.04641073942184448, -1.8299667835235596, 1.983878493309021, 1.0547380447387695, 0.05284987390041351, -2.0358121395111084, 0.8657333850860596, 0.26790180802345276, 0.5089826583862305, -0.44507163763046265, 0.818145751953125, 1.5109299421310425, -2.758458137512207, -0.47864750027656555, 1.4826912879943848, 0.8642376065254211, 0.38782936334609985, -0.47392764687538147, 2.0685324668884277, -0.6326236724853516, 0.13629604876041412, 0.09855373948812485, 1.8502182960510254, 1.372663140296936, 1.5005775690078735, -0.019244004040956497, -0.7841843962669373, 0.5470739006996155, 0.16855354607105255, -0.3468439280986786, -0.5550468564033508, 0.16743497550487518, 0.16183894872665405, -0.7700448036193848, -0.21431581676006317, 0.26299765706062317, -0.5002915859222412, -0.9003662467002869, 1.9566872119903564, -0.21540643274784088, -0.2956090569496155, 0.12140839546918869, -0.6616703271865845, -0.385405033826828, 0.37418463826179504, 0.5979070663452148, -0.7504163384437561, -0.6699677109718323, 0.33266714215278625, 1.257047176361084, 0.20593994855880737, -0.6188715100288391, -0.4950723350048065, -0.5684201121330261, 0.6127104163169861, -1.3364273309707642, 0.2387876808643341, 1.3073418140411377, 0.42629528045654297, 0.8974193334579468, -1.2226921319961548, -2.0156919956207275, -0.9907406568527222, -0.04472002014517784, -0.0839759111404419, -0.20324084162712097, 1.2798877954483032, 0.6925186514854431, 0.3262484073638916, -0.9802460670471191, -0.3892676830291748, -0.33345383405685425, -1.3121488094329834, -0.06313549727201462, -0.027693606913089752, -0.7537089586257935, 0.8181500434875488, 0.5433220863342285, 0.13630130887031555, -1.200516939163208, -1.5977312326431274, 0.6765987873077393, 1.7540196180343628, -0.3993581533432007, -0.27824094891548157, 0.588628351688385, 1.1079226732254028, -0.6022902131080627, 0.6858460903167725, -0.1518576741218567, -0.7607080936431885, 2.022920608520508, -0.9283638596534729, -0.6564986109733582, -2.378774881362915, 2.467947006225586, 0.412062406539917, -0.3638806939125061, -1.6629558801651, -0.5188369154930115, 1.4917067289352417, 1.223726749420166, 1.3679215908050537, 0.37558501958847046, -1.3350919485092163, 0.7649902701377869, -2.294600248336792, -0.11412043869495392, 1.0084673166275024, 0.17683430016040802, -0.37601253390312195, 0.8674424886703491, 0.6763453483581543, 1.564669132232666, 0.20831221342086792, 0.6809841990470886, -0.8741403818130493, 1.1708779335021973, 0.579194188117981, -0.39468467235565186, 0.32616671919822693, -0.40821734070777893, -0.49163755774497986, -0.7184341549873352, 0.9415866136550903, -1.2652320861816406, 1.7107433080673218, 1.4031982421875, 0.28555333614349365, -0.10900231450796127, 0.7447659373283386, -1.2024439573287964, 0.2825888693332672, -1.4168967008590698, 1.5867669582366943, 2.3386685848236084, -0.6925682425498962, 0.6070091128349304, -2.0441908836364746, 1.5450108051300049, 1.3594083786010742, 0.10987112671136856, -0.3004118800163269, -0.3145577013492584, -1.195452094078064, -0.281652569770813, 0.7698760628700256, 0.4096316695213318, 1.6568833589553833, -0.9204958081245422, -0.4469580054283142, -0.07257170975208282, 1.9907721281051636, -0.5706689357757568, -0.9061638116836548, 1.685593605041504, 0.4832109808921814, -0.26083722710609436, -0.5752353668212891, -1.8228731155395508, -0.17364999651908875, 0.8368850946426392, -1.207611322402954, -0.010330108925700188, -0.5888217687606812, -1.4934419393539429, -1.1647863388061523, -1.7703843116760254, 0.5898038148880005, -1.3623570203781128, 1.201884150505066, 0.09696929901838303, 1.375887393951416, 1.4145389795303345, -0.35629528760910034, -0.7173758745193481, 0.09445641934871674, -1.4907140731811523, 0.16407720744609833, -0.6687597036361694, 0.8485363721847534, 0.09920629858970642, 0.813804566860199, 0.7485541701316833, 0.5223167538642883, 1.810009241104126, 0.9375356435775757, -0.2282126098871231, -0.7658786773681641, -0.5837494134902954, -0.6680693626403809, -0.8064934015274048, 0.5225906372070312, -0.391282856464386, 1.644106388092041, 1.2823846340179443, 1.733134150505066, -0.3244090676307678, -1.396247148513794, -1.5719696283340454, 0.47023558616638184, -1.5103672742843628, -0.6747140884399414, 0.7216168642044067, -0.603302001953125, 0.6684723496437073, 1.221399188041687, -0.34309273958206177, -0.8717385530471802, 0.2796808183193207, -0.2758503556251526, 0.964229166507721, 0.0726456418633461, -0.22783337533473969, -1.9462924003601074, -0.8191380500793457, -0.06840153783559799, -0.8471583724021912, -1.367194652557373, -2.1731584072113037, 1.4935709238052368, -0.8073514699935913, -1.6353861093521118, 0.22520826756954193, 1.1404564380645752, -0.14608891308307648, -0.9043601155281067, -1.367977499961853, -0.5143554210662842, 1.4712145328521729, -0.04948645085096359, -0.36181920766830444, -0.030736155807971954, -0.6982525587081909, 1.9751372337341309, -1.2729443311691284, 0.7628546953201294, 0.8689148426055908, 2.8651599884033203, -1.4833872318267822, 0.8620303273200989, -0.37234583497047424, -1.8347797393798828, -0.492754727602005, -0.23729446530342102, 1.4790310859680176, -0.3234419822692871, 1.7683618068695068, -0.18518702685832977, 0.9095686674118042, -1.2872360944747925, -1.4346626996994019, 0.28744208812713623, -1.5751532316207886, 1.4223040342330933, -1.2154501676559448, -0.48585304617881775, -0.19967085123062134, 1.0292824506759644, -0.8064237236976624, -0.07120870798826218, -1.1485588550567627, -1.78838312625885, 0.7558885812759399, -0.4526418447494507, -1.6680595874786377, -0.14168037474155426, 0.4151943027973175, -0.8491824865341187, -0.07103808224201202, -0.49395549297332764, -1.3712917566299438, -0.022873330861330032, -0.4686391055583954, -0.24863137304782867, -0.7408163547515869, 1.8756232261657715, 0.34432458877563477, -1.6454498767852783, -1.7426730394363403, -2.365455150604248, -0.4727504551410675], [0.10044689476490021, -0.3478187620639801, -0.4879388213157654, -1.2983629703521729, 0.33212903141975403, 0.08658398687839508, -0.20554891228675842, -0.7721953392028809, 0.7420523762702942, 0.26394742727279663, -0.6197695136070251, -0.11843650788068771, -0.9330238699913025, 0.4342139959335327, -1.0549283027648926, 1.215196132659912, 0.7437825798988342, -0.17749406397342682, -0.9824522137641907, -0.3607929050922394, -0.30987805128097534, 0.6018469333648682, -0.27945300936698914, 0.31674379110336304, -0.4054577052593231, -2.2209131717681885, -1.1945183277130127, 2.697253942489624, -0.5610177516937256, 1.0487302541732788, -0.33466461300849915, 0.8051088452339172, 0.2211856245994568, 0.93486088514328, 1.374636173248291, 0.197365403175354, -0.1945512741804123, -0.10643572360277176, -1.294272541999817, -1.3701660633087158, -1.9021168947219849, -0.4180760979652405, -1.4632645845413208, 0.6775498986244202, -0.2437705546617508, -0.2223949432373047, 0.09049975126981735, 0.2639015018939972, 0.011228298768401146, -0.6591108441352844, -0.6191885471343994, -0.45729538798332214, 0.1834656149148941, -1.3639788627624512, 0.14153438806533813, -0.3466581702232361, -1.038784146308899, 0.07288103550672531, -0.43076109886169434, 0.11255481839179993, -0.14737476408481598, 0.8742905855178833, 0.07167559117078781, -0.870110034942627, -0.21121403574943542, 2.0961904525756836, 0.7923672199249268, -1.3986467123031616, -1.7753957509994507, 0.6528869271278381, 1.4370150566101074, -0.07069554924964905, -0.8767971992492676, -0.03726448491215706, 1.3758397102355957, -1.4496403932571411, -0.5176501870155334, -0.19425982236862183, 1.1987512111663818, 0.30577754974365234, 0.2733115255832672, 0.7134167551994324, -0.7017749547958374, -0.17507076263427734, 0.24371424317359924, 1.9276905059814453, -0.3250878155231476, 0.2876352071762085, -2.512284517288208, 0.1610972285270691, -0.376013845205307, 0.7507420778274536, 0.1221529170870781, 2.0747931003570557, -0.5479334592819214, -2.050844192504883, 1.7788281440734863, -0.293239951133728, 1.1466299295425415, -0.4752155840396881, 0.5514292120933533, 1.2273952960968018, -1.5240025520324707, 0.748992383480072, -1.3328274488449097, 0.04032865911722183, 0.7333192229270935, -0.8955398797988892, -1.3499414920806885, 0.19216378033161163, -0.14022091031074524, -0.3417823910713196, -0.9972711205482483, 0.29460659623146057, -0.363786906003952, -0.8864448070526123, -0.4639941155910492, -0.901677131652832, -0.6261086463928223, 1.3514528274536133, -0.6028332710266113, -0.15092043578624725, 0.22505420446395874, -0.33041396737098694, 0.04215557873249054, 0.5736129879951477, 0.5744835734367371, -2.0035781860351562, -0.7167820930480957, -0.7331075668334961, -0.8083705306053162, -0.05100700631737709, 0.7941229343414307, 1.102423906326294, -0.21029900014400482, 1.2963167428970337, 0.2394847720861435, -0.3867568075656891, 0.7836273908615112, 0.05046725645661354, 1.3090603351593018, -0.037123098969459534, 0.15174730122089386, -0.5087499618530273, -0.6204755902290344, -0.6810933947563171, 0.6833096742630005, 0.7169041037559509, 0.4119330644607544, -0.5746864676475525, 0.551058292388916, -0.14011958241462708, 0.6261336207389832, 0.5260202884674072, -1.1941388845443726, -2.206843614578247, -1.2032403945922852, 0.6246563196182251, 0.6831793785095215, -0.05958986654877663, -0.10328799486160278, -0.04188067838549614, -0.7757336497306824, -0.7996440529823303, 0.4441015422344208, -0.29113978147506714, 0.38161996006965637, 0.03576089069247246, 0.08047053962945938, 0.00981877651065588, -0.5002442598342896, 0.3586236536502838, 0.4386694133281708, -2.2194528579711914, -0.2540188133716583, -0.31372272968292236, -0.31945550441741943, -0.5451375246047974, 0.3134433329105377, 0.1403534859418869, -0.5342589616775513, 0.09098326414823532, -1.6957954168319702, -1.5070070028305054, -0.9123006463050842, 0.5225862860679626, 0.12403310090303421, -0.7543008327484131, 1.2487573623657227, -1.3099684715270996, -0.283389151096344, -0.2886672914028168, -1.3009631633758545, -0.819110631942749, -1.174730658531189, -1.279214859008789, -1.2597016096115112, 0.14415480196475983, -0.5385122299194336, 0.07670050859451294, -0.24886943399906158, -0.8402007818222046, -0.0030270174611359835, 1.0546458959579468, -1.8378374576568604, 0.4512779116630554, -0.018316209316253662, -0.24543359875679016, 1.3397749662399292, -0.29178375005722046, 0.2624515891075134, -0.2774719297885895, -0.30065861344337463, -0.8962624073028564, -0.3245847225189209, 1.1021690368652344, 0.1530732661485672, -0.1254379004240036, -0.7035576701164246, -2.7141056060791016, -0.10941647738218307, 0.14282569289207458, -0.05348946154117584, -1.3257704973220825, 0.5585078001022339, -2.2609825134277344, -1.3768081665039062, 1.35684072971344, 0.3495667576789856, 1.3232121467590332, -0.7771271467208862, -0.734014630317688, -2.0392661094665527, 1.8629249334335327, -1.6221812963485718, 1.0887895822525024, -1.068825602531433, -0.6632835268974304, 0.22004076838493347, -0.567493736743927, 0.512901782989502, -0.5086120963096619, 0.2035759538412094, -0.024483801797032356, 1.5178741216659546, -0.42610570788383484, 1.492531657218933, -0.956773579120636, 1.3866949081420898, 0.5110140442848206, -0.2406243234872818, 0.2025444060564041, 0.42494770884513855, -0.07818581163883209, 1.0268428325653076, 1.1375144720077515, -0.013151596300303936, -0.13910867273807526, 0.9839438199996948, 0.5833668112754822, -0.053581081330776215, 1.5968282222747803, -0.3640785813331604, -1.5792649984359741, 0.017322149127721786, 0.24762943387031555, -0.3806293308734894, 0.36840370297431946, 1.865734577178955, 3.3135828971862793, -0.6851261854171753, 0.165130615234375, 0.4889219105243683, -0.9138810038566589, 1.6752173900604248, -0.7221303582191467, 0.033764470368623734, 0.31340572237968445, -1.8104455471038818, -0.8742547631263733, -0.3059016168117523, 1.3065452575683594, 1.557538628578186, -0.8078641891479492, -0.5274213552474976, -0.9315218925476074, 0.23181581497192383, 0.26219403743743896, 1.6333222389221191, 1.9139331579208374, 1.0226423740386963, 0.14987759292125702, -0.31917667388916016, 0.2473324090242386, 0.6994604468345642, 0.6023332476615906, -1.0390212535858154, 0.5195680856704712, 1.3198820352554321, -0.14617259800434113], [-0.042667195200920105, 0.8478617072105408, 0.18486648797988892, 0.35479775071144104, -1.5941853523254395, 0.27785682678222656, -0.19988036155700684, 0.5394331812858582, -0.41489970684051514, 0.34838390350341797, -1.2916195392608643, 0.03815603256225586, 0.2955571413040161, 0.5567517280578613, 0.8048641681671143, 0.8230133056640625, -0.23689477145671844, -1.5405044555664062, -0.22850461304187775, -0.6398898363113403, -0.4999106824398041, 1.0713648796081543, -0.5610578060150146, -2.1261467933654785, 0.45078614354133606, 1.1028927564620972, -0.9577621221542358, 0.5252275466918945, -1.6482126712799072, 0.5135819911956787, -1.8705083131790161, -1.731682538986206, 0.25163534283638, -0.9293375611305237, 1.0229532718658447, -1.2040590047836304, 0.9742945432662964, -0.08459647744894028, 0.7545461058616638, 0.9086524248123169, 0.8350439667701721, 1.3732796907424927, 1.3472230434417725, 0.2966706156730652, 1.2404357194900513, 1.0872327089309692, -1.1663843393325806, -1.0835390090942383, 0.5067236423492432, -0.3845272362232208, -0.562142550945282, 1.1428700685501099, 0.9778643846511841, 0.9188143610954285, 1.1092884540557861, 0.8594263195991516, -0.9211312532424927, 0.11962546408176422, -0.3649454414844513, -0.98936927318573, -0.7727678418159485, 0.06988181918859482, 2.5472025871276855, -1.6202456951141357, -1.3874577283859253, 0.7412330508232117, 0.4065820276737213, 0.2676185965538025, -3.440134048461914, -1.008349061012268, -0.6935738325119019, -1.6137665510177612, 1.0245205163955688, -0.5557933449745178, -0.738426148891449, -0.5848246216773987, -0.4341872036457062, -1.642289161682129, -1.074129343032837, 1.9408396482467651, -0.09762465208768845, 0.764792799949646, 0.15219715237617493, 0.9724164009094238, -1.277082085609436, -0.41845616698265076, 1.7378931045532227, -1.256109595298767, -0.9560202360153198, -1.4720112085342407, 0.3580327033996582, 0.9170884490013123, 0.01694158837199211, -0.9516823291778564, 0.8823176622390747, -1.5142090320587158, -0.4567476212978363, -0.9478932023048401, 1.7762720584869385, -0.5308274626731873, -0.6652417182922363, 0.37341439723968506, -0.6407490968704224, 0.39883002638816833, 0.20000536739826202, -0.528713583946228, -0.6424961686134338, 1.8274812698364258, -1.1650941371917725, -0.9893389344215393, -0.3226098120212555, -0.9357689619064331, 0.27173998951911926, -0.08048249036073685, 0.5429945588111877, 1.0046066045761108, -0.8570688962936401, -0.28269723057746887, 0.6603085398674011, -0.7898714542388916, -1.2311917543411255, 1.302174687385559, 0.500674307346344, -2.6752030849456787, -0.0956629142165184, 1.1581377983093262, 1.28864586353302, -0.3377622663974762, -0.501939594745636, 0.9770008325576782, -0.6432857513427734, 0.24071139097213745, 0.03422677889466286, -1.4913690090179443, 1.1044538021087646, -0.1885521411895752, -0.16634081304073334, 0.2663307189941406, 1.5640029907226562, 0.28074148297309875, -0.31404316425323486, -0.844620943069458, -0.06537092477083206, -0.753448486328125, 0.8368663191795349, -0.21286141872406006, -0.05521189421415329, 0.7063013315200806, -0.9546710848808289, 0.00377523060888052, -0.33081603050231934, 0.11399988830089569, -1.1831530332565308, 0.880316972732544, -0.6918382048606873, -1.3168050050735474, -1.2250782251358032, 1.9911224842071533, -0.11395076662302017, -1.4151242971420288, -1.5620300769805908, -0.6396312713623047, 1.0167434215545654, -0.09577195346355438, -0.2764347195625305, -0.2728629410266876, 1.2567293643951416, -2.5640149116516113, 0.1960732787847519, 0.46497631072998047, 1.442313551902771, 0.7382215857505798, -0.44854679703712463, -0.43705183267593384, -0.19318297505378723, 0.3055397570133209, 0.37222251296043396, 0.557269811630249, 2.4144842624664307, 0.6750759482383728, -0.5374256372451782, -0.5363786220550537, -1.2444393634796143, -1.2705466747283936, 0.17536000907421112, -1.2768588066101074, -0.17787247896194458, -0.24825051426887512, 0.08421753346920013, -1.5490212440490723, 0.37158069014549255, 0.48439428210258484, 0.15703551471233368, -1.4468417167663574, -0.22007694840431213, -1.643981695175171, -1.9427510499954224, 0.2746237516403198, 0.5671502947807312, -0.06826749444007874, -0.3119606673717499, 0.18521292507648468, 0.4136894643306732, 1.0687432289123535, 0.3257678151130676, 0.43498310446739197, 0.49855539202690125, 0.4335501790046692, -1.2790905237197876, 0.5914086103439331, 0.8520858883857727, -1.3066954612731934, 0.03849528357386589, -0.19396886229515076, 0.054619528353214264, -0.15640701353549957, 1.3974519968032837, 1.0860391855239868, -0.1309991031885147, 1.8036675453186035, -1.7662054300308228, -0.6074520349502563, 0.1140018105506897, 0.25932011008262634, -0.11534114181995392, -0.4464913010597229, 0.3578922748565674, -0.7653144598007202, 1.163976788520813, 0.49910038709640503, 0.08856038004159927, -0.2104652374982834, 0.5330879092216492, -0.017391880974173546, -0.14975588023662567, -1.3951234817504883, 1.1186426877975464, -1.8903032541275024, -0.3004608750343323, 1.25875723361969, 0.4401349425315857, -0.13083915412425995, 1.6775026321411133, -0.3502048850059509, -0.4772215485572815, -0.6489409804344177, -1.0095123052597046, -0.8785918951034546, 0.23315435647964478, -0.4723506569862366, 0.3627515137195587, 0.3552619516849518, -1.6070005893707275, -0.8600077033042908, -1.525659441947937, 0.2032935917377472, 0.13363680243492126, -0.6268082857131958, -0.43831759691238403, 0.31260764598846436, -0.4483141601085663, -1.9363900423049927, -0.656526505947113, -1.137782335281372, -0.23626402020454407, -0.6293519139289856, -0.6019300818443298, -0.38684284687042236, -0.15938884019851685, -0.11486879736185074, 0.4239858388900757, -0.9168961644172668, -1.6497528553009033, 1.6506805419921875, -0.853312611579895, 0.6779845952987671, 0.8346689939498901, 1.07729971408844, 1.2884585857391357, 2.3271117210388184, -1.9729779958724976, 0.9739909172058105, 0.5307932496070862, -1.699695348739624, 1.0719832181930542, 1.159909963607788, -0.792279064655304, 3.234217882156372, 0.49810969829559326, 3.2362380027770996, -2.041311502456665, -0.04000109061598778, -1.9484195709228516, 0.25833889842033386, 0.6887932419776917, 0.6418384909629822, -0.1270020306110382, -0.47325336933135986, 0.8677865862846375, -0.08596982061862946]], "dtype": "float32"}, {"__nd__": [1.0174846649169922, -0.9401165843009949, 2.2776336669921875, 0.1782108098268509, -0.415079802274704, -1.0624772310256958, -1.122445821762085, 0.623699963092804, 0.9472776651382446, 0.942499577999115, 0.34343430399894714, -1.1793739795684814, 0.08270799368619919, 0.8130640983581543, 1.1855288743972778, -1.1136902570724487, -0.7641136050224304, 1.6236228942871094, -0.40381816029548645, 1.3701345920562744, 0.64048171043396, 0.7738021016120911, 0.49619919061660767, 0.882907509803772, -1.5235791206359863, 0.15916098654270172, 0.36923548579216003, -0.06308630108833313, 1.1361693143844604, -0.3572382926940918, -0.030681859701871872, 0.81495600938797, -1.9125171899795532, 0.7402975559234619, 0.9302128553390503, -0.8171592950820923, 0.6356679201126099, -0.9747630953788757, -0.2148021012544632, -0.2198978066444397, 0.7934865355491638, -0.3069005012512207, 1.411722183227539, 1.5755894184112549, -0.4314381778240204, 0.3642496168613434, 0.4472859501838684, 0.9435837864875793, -1.1622482538223267, 1.4975374937057495, 0.22371900081634521, 0.741740882396698, 1.7071456909179688, 0.7160816788673401, -1.1103668212890625, -0.6428338289260864, 1.3983511924743652, 0.10943464934825897, -0.30791744589805603, 1.0946439504623413, -0.1037125363945961, 1.473787784576416, 1.1690160036087036, -0.6688600778579712, -1.9088594913482666, -2.8870553970336914, 0.5413215160369873, 0.08524041622877121, -1.089682936668396, 0.04304920509457588, -0.2803402245044708, -1.1624069213867188, -1.5838431119918823, 0.8200508952140808, -1.1501704454421997, 0.32577887177467346, 0.3098044693470001, 1.8367704153060913, -0.2922726571559906, -0.777428150177002, -1.0582062005996704, 1.3751624822616577, -0.49569645524024963, 0.4154934585094452, -1.9619888067245483, 0.4666919708251953, 0.16448485851287842, -0.243622824549675, 0.3483789265155792, -0.4879217743873596, 1.0892916917800903, 1.1607071161270142, 0.9965101480484009, 2.0667288303375244, 0.35149574279785156, 0.4796045124530792, -2.735020399093628, -0.12767788767814636, 0.0514901764690876, 0.24947373569011688, 0.6296017169952393, 0.4753482937812805, -0.40608611702919006, 2.1322131156921387, 0.9052618741989136, -0.540748119354248, 0.3796676993370056, -0.4908348321914673, 0.5638675689697266, -0.4386918246746063, 2.2458367347717285, 1.9570566415786743, 2.163783550262451, -0.7691636085510254, 1.415109634399414, 1.1745883226394653, -1.0329296588897705, -0.10512738674879074, 2.184569835662842, -0.43137916922569275, 0.479228138923645, 0.024102289229631424, -0.20942573249340057, 0.42658790946006775, 0.08161609619855881, 0.389102965593338, 0.2749546468257904, 0.6511969566345215, 1.4548766613006592, 2.750490427017212, 0.7641370296478271, 0.459593266248703, 0.4945891499519348, 1.0278241634368896, 2.1276392936706543, -0.9202965497970581, -0.4471758008003235, -0.0477314218878746, 0.5161295533180237, -0.4294854998588562, 0.0741727203130722, -0.41048628091812134, 0.67852783203125, 0.5525729060173035, 0.20839354395866394, 0.19346728920936584, 0.00318347685970366, 1.1905099153518677, 0.40842127799987793, -0.24161626398563385, 0.7202981114387512, 0.9728213548660278, -0.02107643149793148, -1.5868059396743774, -0.9139808416366577, -0.33826518058776855, -1.892772912979126, 0.5612393617630005, -0.004551511723548174, -0.18025943636894226, 0.7138339877128601, 1.6632380485534668, -0.7186489105224609, 0.7143427133560181, 0.6700466871261597, 0.880565345287323, -0.5199840664863586, 0.5403860807418823, -0.5542075634002686, 1.0872796773910522, 1.0964915752410889, 1.5838894844055176, 0.9046698808670044, 2.3464040756225586, -0.6451702117919922, 0.2972208857536316, 1.4909353256225586, 1.3894562721252441, -0.9340565800666809, 2.004260301589966, 0.430467426776886, 0.7060835957527161, 0.8790924549102783, 0.14497096836566925, 0.3663225769996643, 1.5393143892288208, -0.2399924099445343, 0.8479244709014893, -1.8161572217941284, -0.2776103615760803, 0.5722411274909973, 1.3981101512908936, -1.095391869544983, -0.5384729504585266, 0.40300360321998596, 1.4868102073669434, 1.6015305519104004, 0.43045592308044434, 0.11195932328701019, 1.0986127853393555, -0.21106038987636566, -2.1364591121673584, 1.0021514892578125, 2.0446763038635254, -1.1546117067337036, 0.3144510090351105, 1.6440802812576294, 1.2685290575027466, 0.9685254693031311, -0.27273622155189514, -0.7229041457176208, 1.8803797960281372, -0.6274248957633972, -1.2627557516098022, -0.7389742732048035, -0.6062785983085632, -1.0390225648880005, 0.9168826937675476, 0.547314465045929, -0.28677669167518616, -0.618074357509613, 0.19361574947834015, 0.03083689883351326, -0.7856568694114685, -2.3562660217285156, -0.18905584514141083, -0.9099651575088501, -1.1868637800216675, -0.6597002148628235, 1.5837938785552979, -0.439793199300766, 0.6024520397186279, -0.4025200605392456, -1.4475620985031128, 1.209861397743225, -1.887813687324524, 0.6968799233436584, 1.308713674545288, -0.7487467527389526, -0.825298011302948, 1.0927850008010864, 0.9161752462387085, 0.16228778660297394, 1.3370237350463867, -0.566859245300293, 0.1186479851603508, 1.9173856973648071, -1.2164173126220703, -1.3471288681030273, 0.20938076078891754, 0.7405499219894409, -0.4016035795211792, -1.6365586519241333, -0.3235090970993042, -1.0186007022857666, 0.46845895051956177, -0.27214404940605164, -0.035742901265621185, 0.5674737095832825, -0.2018672674894333, 0.2058722823858261, -1.2877522706985474, -1.657007098197937, 0.25482070446014404, 0.36143556237220764, 0.21031023561954498, -0.58017897605896, 0.9195871949195862, 0.5798044204711914, 0.0006447675405070186, 0.504012942314148, 0.46887895464897156, -0.2009378969669342, -1.368086814880371, -0.1477975696325302, -0.7268801927566528, -0.6632513403892517, 0.22138671576976776, -1.0311511754989624, 2.104570150375366, -2.131021499633789, 0.32165658473968506, -0.01645585149526596, -1.7754201889038086, -0.36366575956344604, 1.589576244354248, -0.7317567467689514, 0.4637618064880371, 1.6552770137786865, 0.036352530121803284, -0.3782626986503601, -3.384148359298706, 0.09872923046350479, -1.4472401142120361, 0.42055514454841614, -0.3021969497203827, -0.606120228767395, 0.9153873920440674, -1.8023905754089355, -0.590060830116272], "dtype": "float32"}, {"__nd__": [{"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}, {"__f__": "nan"}], "dtype": "float32"}, 1.0, 0.0], "e": {"__nd__": [-13.93213176727295, 67.57310485839844, -6.402176856994629, 5.148228645324707, -19.99140167236328, 22.166147232055664, 5.589848041534424, 11.162564277648926, -9.423591613769531, 15.412443161010742, 8.164199829101562, -33.73814010620117, -10.676609992980957, 25.505064010620117, -0.605334460735321, -0.9652261137962341, -4.62076473236084, -24.856901168823242, 24.629108428955078, -29.17299461364746, 12.466155052185059, -6.579919338226318, 24.843141555786133, 38.369876861572266, 25.470273971557617, -5.766775131225586, 7.575436115264893, -5.953657150268555, -37.24846649169922, 6.035172462463379, -15.239946365356445, -27.276350021362305, -19.476259231567383, 9.351092338562012, 7.104441165924072, 7.261571407318115, -11.020514488220215, -15.772672653198242, 4.945140361785889, 7.853627681732178, -21.199316024780273, 8.095117568969727, 16.8126277923584, -0.7762987613677979, -2.475132703781128, 31.920583724975586, -12.54976749420166, 1.0111074447631836, 16.392799377441406, 0.10149691253900528, 32.10503005981445, -6.78606653213501, 19.055923461914062, 10.425569534301758, 14.398908615112305, 22.52060317993164, -44.356075286865234, 27.87185287475586, -32.149871826171875, 14.788037300109863, -10.101041793823242, -21.247943878173828, -24.391996383666992, 48.623077392578125, -9.325881958007812, 3.5816662311553955, 17.65024757385254, -2.745555877685547, -24.730981826782227, 16.398935317993164], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.8341156244277954]], "dtype": "float32"}, {"__nd__": [1.46615469455719], "dtype": "float32"}, {"__nd__": [-1.174454927444458], "dtype": "float32"}, 0.5, 0.5], "e": {"__nd__": [-1.1986987590789795], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.1659523248672485, 0.17281965911388397, 1.357035517692566, -0.6684601902961731, -0.18740205466747284, -0.1968642920255661, -0.7153116464614868, -0.07311072945594788, 0.8394386768341064, -0.7157764434814453, 0.005587398074567318, -2.106229543685913, -1.7780247926712036, 0.46974003314971924, -0.6840694546699524, -0.8798176646232605, 2.002037763595581, -1.2684993743896484, -0.3485053777694702, -1.909993290901184, -2.748253107070923, 1.2994129657745361, 1.0286548137664795, -0.6923625469207764, -0.271443247795105, 0.1275739073753357, -0.4782831370830536, 0.4588198959827423, -0.45536521077156067, 0.10013952851295471, -0.975527286529541, 0.33003515005111694, -0.4897392988204956, -1.362187385559082, 0.3001834452152252, -0.1660659909248352, -0.13294167816638947, 1.3183391094207764, 1.2662187814712524, 0.4864118695259094, -0.7191991209983826, -1.1931105852127075, -0.4668314456939697, 0.19158995151519775, 2.305079936981201, 0.6115590929985046, -1.173224687576294, -1.2527177333831787, -1.8010984659194946, 1.0098929405212402, 0.8952929973602295, 1.6431083679199219, -0.593578577041626, -1.2533068656921387, 0.28362035751342773, 0.35816875100135803, -1.7735207080841064, 0.01574351079761982, 0.2783665060997009, -0.5075357556343079, 0.33915477991104126, 1.6221615076065063, 0.40579667687416077, 1.078195571899414, -1.2182748317718506, -0.14487268030643463, -0.2431955188512802, 1.6603118181228638, -0.31186625361442566, -0.6231529116630554, 0.2623477280139923, -0.1750122308731079, 0.08632997423410416, 1.1811968088150024, 0.181158646941185, -0.1034582257270813, 1.4615589380264282, 0.6727476119995117, 0.2725776731967926, 0.5197118520736694, 0.20134681463241577, 0.6659179925918579, -0.30092036724090576, 1.360548496246338, 1.6393449306488037, -0.11379235237836838, -0.7589511275291443, -0.5373848676681519, 0.38695982098579407, 0.3753209710121155, -1.513890266418457, 0.2403501570224762, -1.177191972732544, 2.6433324813842773, -1.4378501176834106, -1.3601478338241577, -0.9249329566955566, -0.06677649170160294, 1.949326515197754, -1.3907063007354736, -0.4165659248828888, -2.5613608360290527, 0.40842756628990173, -3.1884143352508545, 0.7649814486503601, 0.8094513416290283, 1.490523338317871, -1.6655267477035522, -0.1621863842010498, -1.1599054336547852, 1.248008370399475, -0.22032463550567627, -0.5226761102676392, 0.05187975615262985, 1.4309964179992676, 0.5028697848320007, 0.9216887354850769, 1.4999878406524658, 1.6275825500488281, -0.11838990449905396, -1.11934232711792, -1.334317684173584, 0.6795939803123474, -0.22825558483600616, -1.0543973445892334, -1.2257497310638428, 0.6012237071990967, 0.16500480473041534, 0.2857911288738251, 0.28120216727256775, 0.3779458999633789, -0.025893449783325195, 0.2706681191921234, 0.3192251920700073, 0.29113608598709106, -0.41102755069732666, 0.8651091456413269, -0.7079921364784241, 1.174461007118225, 0.7981804609298706, -1.9723964929580688, -1.5093244314193726, 0.9952934384346008, 0.3953501284122467, 0.6953001618385315, -1.041235327720642, -0.7583792209625244, -0.010231506079435349, -0.5519072413444519, -0.9290332198143005, 1.2782083749771118, 0.38255375623703003, 1.4480217695236206, -0.15593977272510529, -0.3686598837375641, 0.6252322793006897, 1.3732613325119019, -0.3794858753681183, -0.9822618365287781, -0.0682123601436615, -0.29327645897865295, -0.08660740405321121, -0.7561131119728088, 0.0490599125623703, -0.23436331748962402, -0.5844802260398865, -0.4333515167236328, 1.2837203741073608, 1.6840648651123047, -1.2043238878250122, -0.5742001533508301, -0.8362956643104553, -2.404149293899536, 0.43814146518707275, 1.4002468585968018, -1.1218205690383911, 0.4700844883918762, -0.2183939516544342, 0.47346433997154236, -0.3110864758491516, -2.0898215770721436, 0.9340508580207825, -1.3922619819641113, 0.11729662120342255, -1.148325800895691, -0.9654035568237305, -0.1207747608423233, -0.39104005694389343, -1.2326247692108154, 0.4566307067871094, -0.916543185710907, 0.6756247878074646, 0.614176332950592, 0.2533423602581024, -1.0550620555877686, 0.6541348695755005, 0.3179583251476288, -1.519182562828064, -0.07007844001054764, 0.35420918464660645, 0.8850629925727844, -0.3000562787055969, -0.2555557191371918, -1.464747428894043, -1.137747883796692, -1.3873884677886963, 0.810010552406311, -0.5514004826545715, 0.503961980342865, -1.5722811222076416, 0.1180199533700943, -1.9317350387573242, -0.8123653531074524, 1.4189658164978027, -0.8570128083229065, 1.0415656566619873, -0.6891797780990601, 1.8874796628952026, -0.16478024423122406, -0.020974252372980118, 0.284518301486969, -0.2732357382774353, -1.5340689420700073, 0.8432701826095581, -2.5783355236053467, 0.46032509207725525, -0.6969760656356812, 0.3934412896633148, -0.5548772811889648, 0.2559925615787506, 0.09435813128948212, -0.9738270044326782, -0.6854380369186401, -1.1848483085632324, 0.6419658660888672, 0.8088799715042114, -0.6156997680664062, 0.4687759280204773, -2.210270643234253, -1.5634193420410156, -0.5605048537254333, 0.701754629611969, -0.17598780989646912, -1.9689608812332153, 0.2416628897190094, 0.8568580150604248, 0.7007513046264648, -0.5746075510978699, 0.04114456847310066, 0.09665530174970627, 0.15933255851268768, -0.8639783263206482, 1.4287497997283936, -0.24787572026252747, -2.4507803916931152, -0.711974561214447, 0.4087647497653961, 0.9345841407775879, -0.9181996583938599, 0.32562172412872314, 2.2084543704986572, -1.7646979093551636, -1.0894492864608765, 0.23849716782569885, 0.04282236099243164, -0.3795551359653473, 1.4726313352584839, 0.7202620506286621, -0.5031042098999023, 0.9102163910865784, 1.1026396751403809, -0.01773333176970482, -1.6226191520690918, -0.4940398931503296, 0.5288936495780945, 0.3919021189212799, -1.282675862312317, -0.49922338128089905, -0.11052727699279785, -0.8778813481330872, 0.4415286183357239, 0.5145348310470581, -2.2002193927764893, 0.1953575760126114, -1.1714118719100952, -0.4254087507724762, -0.9673453569412231, 1.5167137384414673, 0.9412183165550232, 1.3091202974319458, 2.2979705333709717, -1.1182401180267334, -0.5682869553565979, -1.032027006149292, -0.44448089599609375, -0.20437969267368317, 1.3782705068588257, 1.8764053583145142, -0.0988229364156723, 0.002039704704657197, -0.4289751946926117, 2.261744737625122, 1.3856635093688965, -0.5525922179222107, -0.5369253754615784, 0.3634799122810364, 0.7267934679985046, 1.109288215637207, 1.9756550788879395, -0.07187667489051819, 0.6366048455238342, -0.1579795777797699, 0.7483969926834106, 0.9959196448326111, 1.4653968811035156, 0.09760388731956482, 0.7383379340171814, -1.2969307899475098, 1.194968819618225, -1.3501757383346558, -0.4523264169692993, -0.8524585366249084, -0.6899834871292114, -0.08467923104763031, -0.9065690040588379, -0.49504354596138, -0.8813807368278503, -0.4606935679912567, -2.0145530700683594, -1.1480891704559326, 0.3992258906364441, 1.2972520589828491, -1.259613037109375, 0.3769359886646271, 0.7142108678817749, 1.787161111831665, 0.2881917357444763, -2.2451534271240234, -1.0990387201309204, 1.9030100107192993, -0.5770513415336609, -0.03226935863494873, -1.4186041355133057, 0.2040281891822815, -1.2211753129959106, -0.6972974538803101, 1.4330520629882812, -0.40931546688079834, 0.4300486445426941, -0.23087932169437408, -1.137088656425476, -0.45534437894821167, 0.03553112596273422, 0.3122025430202484, -0.904129683971405, 0.11671131104230881, 1.182363510131836, -1.2501296997070312, -0.9580526947975159, -1.252139925956726, 0.8412763476371765, -0.538930356502533, -0.16903479397296906, -0.7351856827735901, 1.0323177576065063, -0.6634065508842468, 1.0948659181594849, 1.2634148597717285, 0.5556091666221619, 0.3375176787376404, -0.5450153350830078, -0.8595617413520813, 0.9321255683898926, -0.4822389483451843, 1.9076201915740967, 0.77853924036026, -0.2239867001771927, 0.03973105549812317, 0.1734962910413742, 0.1067681834101677, -0.520898699760437, -0.5465616583824158, 0.1300184428691864, 0.4591405391693115, 0.002778485417366028, 0.9850597977638245, 0.08344562351703644, 1.729681372642517, -1.2833709716796875, 0.5240558981895447, 0.7986553907394409, 0.23499110341072083, -0.11329110711812973, -1.8030177354812622, -1.3607620000839233, -0.39139649271965027, -0.0929340124130249, 0.23812462389469147, 1.1796268224716187, -0.322858065366745, -1.3911802768707275, -0.38691961765289307, -1.0244669914245605, 0.32617050409317017, 0.14057788252830505, -0.19573265314102173, 0.3386577367782593, 1.4480953216552734, -0.350570410490036, -1.4741132259368896, -0.0138957928866148, 0.3360424339771271, 0.5879687070846558, 0.4413480758666992, 1.5213462114334106, -0.03581670671701431, -0.08382643759250641, 1.3059543371200562, -1.316422462463379, 0.2844725251197815, -1.2872321605682373, -0.07012119144201279, 1.6143262386322021, -0.4808369278907776, 0.8119329810142517, 0.9708935022354126, 0.03247472643852234, -0.36816278100013733, -0.4439942240715027, -0.09092637151479721, -1.0067120790481567, 0.10217636078596115, -1.7697558403015137, 2.3355112075805664, -0.8033056855201721, -0.492031455039978, 0.3619598150253296, -0.5446013808250427, 0.7365888953208923, 0.18743140995502472, -0.8864973783493042, 0.6156597137451172, 0.7218584418296814, -0.16855552792549133, 1.3520821332931519, 0.3223095238208771, 1.0285347700119019, 2.6215908527374268, -0.6770192384719849, -0.7604687213897705, 1.5874682664871216, 2.215568780899048, -1.3965256214141846, 1.1369316577911377, -1.3231608867645264, -0.935048520565033, 0.758948802947998, 0.30742451548576355, -0.4955141842365265, 0.16421927511692047, -0.6306271553039551, -0.8867957592010498, 0.4600267708301544, -0.05901344120502472, -0.10374858230352402, 0.5032154321670532, 0.918329119682312, -1.2107324600219727, 0.5140379667282104, 0.11273251473903656, 0.06286706775426865, -1.0970524549484253, -0.2911408543586731, -3.2276766300201416, 0.3500801920890808, 0.4667472243309021, 0.0025988873094320297, 2.29752779006958, 0.4808143973350525, 0.379936158657074, 0.16308778524398804, -0.6282219290733337, -2.05492901802063, -0.5604413151741028, 0.21329540014266968, -1.0875283479690552, -0.08627577126026154, -0.26797839999198914, 1.3510452508926392, -0.8669082522392273, 0.279153436422348, -0.6319472789764404, -0.23070064187049866, -0.22990275919437408, -0.6627107262611389, -0.7385014295578003, -0.9330488443374634, 0.15203535556793213, -1.1705143451690674, 0.19755969941616058, -0.28079062700271606, -0.9439078569412231, 0.12528418004512787, -0.09201180934906006, 0.4804657995700836, 0.9874082207679749, -1.8613905906677246, -0.14300556480884552, -0.14034916460514069, 0.6262916922569275, 1.6332449913024902, 1.0833275318145752, -1.0182501077651978, -0.7326385974884033, 0.6143900156021118, -0.041934601962566376, 0.3789723217487335, 0.19710779190063477, -1.306926965713501, -0.2573961913585663, 0.00215522525832057, -0.36705729365348816, 1.0015159845352173, -1.32151198387146, -0.7382911443710327, -0.2611556649208069, 1.1962004899978638, -0.35285067558288574, -0.8265141248703003, -1.1486103534698486, -0.0377541184425354, -1.6529994010925293, 0.3680180013179779, -2.841766357421875, 0.8504741787910461, -1.3512355089187622, 0.2280135452747345, 1.4634405374526978, 0.21845825016498566, -1.9455243349075317, -0.6371413469314575, 0.34059634804725647, -1.1134389638900757, 0.8018165230751038, -1.0209624767303467, 0.2153039574623108, -0.6896063685417175, -1.1934707164764404, -1.26108717918396, -1.6834090948104858, -0.6992740035057068, 1.3666162490844727, 1.453371286392212, 0.6987263560295105, 0.799340546131134, -1.0710079669952393, -0.2723083794116974, 0.17435228824615479, -0.5327000021934509, 0.41453832387924194, -1.081136703491211, -0.747919499874115, 0.3537769615650177, -0.039605338126420975, -0.6772579550743103, -0.5864046216011047, -0.7863723635673523, -1.7268168926239014, -0.42845484614372253, 0.425411581993103, -0.062396857887506485, -0.11060218513011932, -1.5285592079162598, 0.32678303122520447, 2.0182480812072754, 1.113919734954834, -0.4345691502094269, 0.898232102394104, -1.2077399492263794, -2.9312472343444824, -0.47971999645233154, 0.018005402758717537, -0.12447839975357056, -0.2134878784418106, 0.30148908495903015, 0.5236355662345886, 0.6630058288574219, -0.2322528064250946, 0.772633969783783, 0.43884220719337463, 0.45474886894226074, 0.617173969745636, 0.245220348238945, 0.18711985647678375, 1.3120609521865845, 0.5501754283905029, -0.34613487124443054, 0.26619404554367065, 0.380349725484848, 0.6063938140869141, -1.6677063703536987, -0.25586584210395813, -0.487888902425766, -0.5365229249000549, -1.0772452354431152, 0.21836207807064056, 0.521550178527832, -1.0699551105499268, -1.0874834060668945, 0.6270978450775146, -0.3602663576602936, 0.685343861579895, 0.47625547647476196, 0.4950147569179535, 0.8784655928611755, -0.01695888862013817, 0.10585261881351471, -0.45115700364112854, 1.067710518836975, 0.5992391109466553, 1.420745611190796, 0.24894845485687256, 0.4556021988391876, 0.3165974020957947, 0.17683811485767365, -1.4770280122756958, 0.5884554386138916, -0.457816481590271, -0.7389166951179504, 0.22760391235351562, 0.20789620280265808, 1.0242540836334229, -1.166183352470398, 0.6213386058807373, -0.14967428147792816, -0.48457175493240356, -0.7482376098632812, 1.1505928039550781, 0.6563467979431152, 0.6330140233039856, -2.9450695514678955, -0.5690470933914185, -0.14476077258586884, 0.8274862170219421, -0.12930385768413544, -0.28246623277664185, -0.10491658747196198, -1.4643774032592773, 1.0150724649429321, 1.4649994373321533, 1.1607587337493896, 0.4559755325317383, -0.06464822590351105, 0.8120232820510864, 1.6821812391281128, -1.5049664974212646, -0.5772491097450256, -0.2573499083518982, -0.5933979153633118, 1.0160282850265503, -0.5264697074890137, 0.45692482590675354, -0.03873806074261665, -0.20043325424194336, 0.4484184980392456, 3.692950963973999, -0.5648547410964966, 0.7993428707122803, -1.1500643491744995, -0.12461051344871521, -0.3903542757034302, 0.30010986328125, -1.6721892356872559, -1.4576175212860107, 0.23315218091011047, -0.1527121365070343, 0.6668058037757874, 0.5654329657554626, 0.7120469212532043, -1.368324637413025, -0.2349006086587906, -1.4375160932540894, 0.0755278691649437, 0.5692844390869141, 0.6529475450515747, 0.5267798900604248, 1.1028393507003784, -0.9397519826889038, -0.7851831316947937, -1.123874306678772, -0.37304800748825073, -0.5672410726547241, 0.8678669333457947, -1.851676344871521, -1.18878972530365, -0.9960622787475586, 0.3336303234100342, 0.5350761413574219, 0.5658110976219177, 0.8937568664550781, -0.925804078578949, 0.904960572719574, -0.05173557996749878, -0.4494987428188324, -1.9249018430709839, -1.0790085792541504, -0.6045238375663757, -0.7748056650161743, 0.07283154129981995, -1.0206202268600464, -1.090562343597412, -3.5498571395874023, -1.5576261281967163, 0.46393176913261414, -0.8241221904754639, -0.16604186594486237, 0.3689083158969879, 1.872927188873291, 1.3501152992248535, -0.17191870510578156, -0.5527104139328003, 0.9726272225379944, 0.35490185022354126, 0.37704646587371826, 0.07298965007066727, 0.23823615908622742, 0.09959153831005096, 1.0321176052093506, -0.5045382976531982, 1.0574361085891724, -0.9467187523841858, 0.7904576659202576, 1.4315382242202759, -0.8154966831207275, -0.06680243462324142, -0.09264347702264786, -0.45431604981422424, -0.23391374945640564, -0.14766205847263336, 0.4331004321575165, 0.7318612337112427, -1.4591429233551025, 0.5520704388618469, -0.32872721552848816, 1.1652880907058716, -0.6758282780647278, -0.43074095249176025, 0.040345318615436554, -0.10549941658973694, 0.9116812944412231, 0.7736997604370117, -1.3126592636108398, -0.2940058410167694, -0.32163769006729126, 1.1249498128890991, -0.251715749502182, 0.7085205316543579, 0.9186103940010071, -1.0772230625152588, -0.1623154729604721, -0.15352538228034973, 0.4645090401172638, 0.31958359479904175, 1.8303440809249878, 1.041368842124939, 0.5695114135742188, -0.22700241208076477, 0.48613646626472473, 0.5431121587753296, -0.3159918785095215, -0.013603376224637032, -0.29467061161994934, -1.263354778289795, -2.018756628036499, 0.31252971291542053, 0.20471836626529694, 0.16058291494846344, -0.348063588142395, -0.6882091760635376, 0.8215265870094299, 0.40321335196495056, -0.6503491401672363, -0.8359650373458862, 1.013572096824646, 0.04890778288245201, 1.2238225936889648, -0.7381408214569092, 0.8367395997047424, -2.9315781593322754, -1.1808565855026245, -0.372917115688324, 0.1496649533510208, 1.0653884410858154, 0.7888377904891968, 1.000635027885437, -2.2901883125305176, -0.4866200387477875, 0.07607632130384445, 2.3293991088867188, 0.2827973961830139, -0.69626784324646, -0.33250948786735535, 0.6038674712181091, 0.2588594853878021, 0.9262663125991821, -0.21226638555526733, 2.1419272422790527, 0.12502416968345642, -0.339101642370224, 1.131305456161499, -1.378440022468567, -0.33396556973457336, 0.2855518162250519, 0.8386487364768982, 1.3064794540405273, 0.5647374391555786, 0.8573479652404785, 0.5423442125320435, 0.14612995088100433, 2.281341075897217, 0.20556925237178802, -0.37515076994895935, 0.5179210305213928, -0.3954331576824188, -0.9550496935844421, 0.20501741766929626, 1.370018720626831, 2.282407283782959, 0.4899672567844391, 0.372467577457428, -0.7515757083892822, 1.0734254121780396, -2.44704532623291, 1.030487060546875, -0.2909482419490814, -1.073638916015625, -0.3894512355327606, -0.3570910394191742, 0.6921175718307495, -1.3206411600112915, -0.15523061156272888, 1.246415376663208, -0.15337702631950378, 0.4623563587665558, -0.6577224135398865, -0.7208447456359863, 0.43781861662864685, 0.619201123714447, -1.0318574905395508, 0.539112389087677, 0.9634707570075989, 0.08409669250249863, -0.5250353217124939, 2.073549270629883, -1.1582235097885132, -0.42867735028266907, -1.4924194812774658, 0.8316918611526489, 1.0699965953826904, 1.1523773670196533, 0.6905771493911743, 0.3550949990749359, 0.7859065532684326, 1.3553547859191895, -1.8520947694778442, 1.1167806386947632, 0.5878810286521912, 1.4736469984054565, 0.6700385212898254, 0.15444715321063995, -1.8529340028762817, -0.28833386301994324, 0.362130731344223, -0.33035212755203247, -0.39830607175827026, -0.9227476716041565, 1.2256572246551514, -0.30181828141212463, -0.8981192708015442, 0.22661325335502625, 0.673316240310669, -0.2984858751296997, 2.0098443031311035, 0.7401336431503296, -0.8196991682052612, 0.13683544099330902, -0.9901072978973389, 0.7447418570518494, -0.5967702269554138, -0.06449954956769943, 0.3756161630153656, -1.1783229112625122, 0.5283778309822083, -1.2496808767318726, -0.3937055766582489, 0.12369780987501144, 0.4759984314441681, -0.2871103286743164, 1.9429171085357666, 0.4960148334503174, -0.30233821272850037, -0.018477773293852806, 0.8565073609352112, 0.33259859681129456, -1.540808081626892, -0.9008873701095581, 1.6650891304016113, -0.7136551141738892, -0.46342933177948, 0.6909610629081726, -1.472448706626892, -0.06640759110450745, 0.019529640674591064, -0.24569900333881378, 1.6537065505981445, -0.6676352620124817, 1.79073965549469, -2.328547954559326, -2.1066999435424805, 1.86887788772583, 1.2561036348342896, -0.5430636405944824, 1.7397257089614868, -1.177210807800293, 0.017474103718996048, -2.0680809020996094, 0.22065427899360657, 0.4415954649448395, -0.8542034029960632, -1.4235776662826538, 0.4890919029712677, 0.8869360685348511, -0.13196788728237152, 0.5353193879127502, -0.39078041911125183, 0.9262370467185974, -1.0146445035934448, -0.9027130007743835, 0.6900017261505127, 0.6617209315299988, 0.25187310576438904, -1.6310597658157349, -0.01962624303996563, -0.5308570861816406, 0.023126285523176193, -0.9678332805633545, 1.3106905221939087, -0.7921177744865417, 0.6049911975860596, -0.3215036988258362, 0.4865066707134247, 0.1724543422460556, 0.3453235328197479, -0.44704538583755493, 0.705264151096344, -0.7071805596351624, 1.8467352390289307, -0.031116358935832977, -0.5779337882995605, -0.03094242513179779, 0.37003615498542786, -2.682798147201538, -0.5007445812225342, -1.1924378871917725, -0.9172649383544922, -0.9213067293167114, 0.9740263223648071, 0.13181205093860626, 0.0759560763835907, -1.4784200191497803, 0.16894863545894623, -0.15642836689949036, -0.12873636186122894, 0.006398181430995464, 2.0637240409851074, 0.217180535197258, 0.1523892730474472, 0.05556178838014603, -0.9612411856651306, 0.9109379649162292, 0.10486745834350586, -0.3836815655231476, -0.045623112469911575, 0.1387837678194046, -0.10170897841453552, -0.0017535766819491982, -1.0912902355194092, -0.2189195603132248, -0.5462111830711365, -0.2730834186077118, 1.0058670043945312, -0.20871801674365997, 0.38835257291793823, -0.6651220321655273, 0.3917281925678253, 0.9081729650497437, -0.5474727749824524, 0.010632535442709923, 0.5252454280853271, -0.9485843181610107, 0.8315938115119934, 0.387798011302948, -0.3482779860496521, -1.949116587638855, 1.1690127849578857, -0.017807302996516228, -0.4820460081100464, 0.47535601258277893, -0.030546603724360466, -1.7386854887008667, 2.068213701248169, 1.2824057340621948, 1.1760194301605225, 1.9533506631851196, 0.6902543306350708, 0.12074178457260132, -1.4448583126068115, -0.1562112271785736, 0.5976740717887878, -0.2685917615890503, 0.3756590485572815, -0.24238164722919464, -0.8491516709327698, -0.11383286863565445], [1.515824317932129, -0.03709841147065163, -1.280314564704895, 1.4337084293365479, 0.8729000091552734, 0.33209291100502014, 0.8008162975311279, 0.371773898601532, -0.05848712846636772, -0.90936678647995, 0.0859472006559372, -0.2950953543186188, 1.8661949634552002, 0.37367910146713257, -0.4620797336101532, -0.4581202566623688, 0.661938488483429, 2.2394607067108154, 0.9229943752288818, 1.4253425598144531, 1.8537904024124146, -0.07175672799348831, 0.37554046511650085, -1.4563580751419067, 0.2802407741546631, 0.4308498501777649, 0.3891201615333557, -0.274728387594223, -0.8203150629997253, -0.7493245005607605, -1.209875226020813, 0.9344296455383301, -0.0023839385248720646, -0.23189379274845123, -0.1513855755329132, 0.686819314956665, 0.2821754813194275, -0.16381579637527466, -0.4256739318370819, -1.4155789613723755, -0.8628500699996948, 0.67192143201828, 0.9515715837478638, -0.36970585584640503, -0.20504654943943024, -0.14974111318588257, 0.5816004276275635, 0.5681499242782593, -0.45735177397727966, -0.48916923999786377, -0.15120968222618103, 0.05149092897772789, 0.3492527902126312, -1.616967797279358, 0.5106310844421387, -0.40963324904441833, -0.8519388437271118, 0.16940177977085114, -0.13210223615169525, 0.4591931998729706, -0.7356494069099426, -0.7724745273590088, -0.5935033559799194, -1.3289722204208374, 0.18667112290859222, -0.05334191769361496, -2.2408180236816406, 0.6418708562850952, -0.6022404432296753, -1.987754464149475, -0.6598525047302246, -1.6565501689910889, -1.5640965700149536, 1.2389755249023438, 0.22616347670555115, -0.2737179696559906, -1.3600037097930908, 1.3004870414733887, 0.3936329483985901, -1.1995024681091309, 1.188256025314331, -1.8219691514968872, 2.5512218475341797, 1.3411611318588257, -2.1244161128997803, -0.8939773440361023, 0.5268610119819641, 0.47462499141693115, -0.6967388391494751, 0.3017398715019226, 0.10606792569160461, -0.8257228136062622, 0.23091688752174377, 0.6018146276473999, 0.8416564464569092, 0.6962324380874634, 0.36201179027557373, -0.7002468109130859, 0.5148072838783264, -0.9901477098464966, -0.5903307199478149, -1.6940141916275024, 0.8742334842681885, -1.4255321025848389, -2.5966060161590576, -1.3398135900497437, -2.193185567855835, -0.5760765075683594, -1.7692070007324219, 0.3572428822517395, 0.5127962231636047, 0.05509914085268974, -0.10660631954669952, -1.2833688259124756, -0.2822973430156708, -0.16080176830291748, -0.7508251667022705, -1.5374271869659424, 0.45968183875083923, 0.5627648234367371, -0.2638750672340393, -1.2533469200134277, -1.023263931274414, 1.203270435333252, 0.5485302209854126, -0.1635773777961731, -0.04228672757744789, 1.1805920600891113, 0.10935037583112717, 0.09988035261631012, -1.0253095626831055, -0.5011132955551147, -0.3229276239871979, 0.024734895676374435, -1.378667950630188, 2.011326551437378, -1.3723196983337402, -1.7052615880966187, -0.6568154096603394, 2.113692283630371, 0.6949884295463562, 0.33734607696533203, 1.1143783330917358, -0.7261115908622742, 0.21760639548301697, -1.4734227657318115, 0.06244303286075592, 0.4171271026134491, -0.9640666842460632, -1.654750108718872, -0.38561633229255676, -0.2618812322616577, -0.36966803669929504, 0.9213914275169373, -0.5017381906509399, -2.0355842113494873, -0.7640594244003296, -1.2672969102859497, 0.7510894536972046, -0.9540826678276062, 1.0482456684112549, -0.8707008361816406, 1.3642855882644653, -0.6906490921974182, -0.1835380643606186, -0.5251978039741516, 1.5061542987823486, 0.407360315322876, 0.5492624044418335, -1.8867393732070923, -0.2044815570116043, -1.4813196659088135, -0.24773089587688446, 0.14848490059375763, -1.9030835628509521, -2.3311238288879395, -0.1348680704832077, -0.5743184685707092, 2.1796586513519287, 0.6873594522476196, -1.5103657245635986, 0.19791831076145172, 0.09423583000898361, 1.28151535987854, -1.61027991771698, -0.6250854730606079, 0.7398783564567566, 0.45436999201774597, 0.3184942305088043, -1.1215226650238037, -1.1988383531570435, -1.1727912425994873, -1.1541434526443481, 0.8105217218399048, -0.5492709279060364, 2.379167079925537, 1.714634895324707, 0.9721632599830627, -0.20685958862304688, 0.5721380710601807, 1.0680776834487915, -0.14896608889102936, 0.17302849888801575, -0.7126809358596802, 0.6368629932403564, -0.24421630799770355, -0.44115105271339417, 0.28678828477859497, 0.6746766567230225, -1.7402414083480835, -1.1274300813674927, -0.7481803894042969, -0.2800672650337219, 1.0893349647521973, 0.3232778310775757, 1.6549855470657349, -0.3460552990436554, -0.23643963038921356, 0.6416562795639038, 1.0136903524398804, 2.5226969718933105, 2.092938184738159, -1.4085745811462402, 0.8894333839416504, 0.46210166811943054, -0.7565924525260925, -0.7438773512840271, 0.741134762763977, 2.135617256164551, -0.08101136237382889, 0.05026058480143547, -0.22908586263656616, 0.34388795495033264, 1.2164032459259033, 0.3115479350090027, -1.0339850187301636, -0.16716066002845764, 0.6208367347717285, -1.0531940460205078, 1.7083827257156372, -1.3560305833816528, -0.631843090057373, 0.6517419219017029, -0.4516368806362152, -0.30302128195762634, -1.9244383573532104, -0.633194088935852, -0.3049387037754059, 2.2546913623809814, 0.7164360284805298, 0.3932800590991974, 0.14570267498493195, 0.03947089985013008, -0.17116646468639374, 0.717780351638794, 1.4681892395019531, -0.6215955018997192, 0.09250535070896149, -1.1184219121932983, -0.6094433069229126, -0.03122480772435665, -0.10799895226955414, 0.08790700882673264, 0.4137405753135681, -1.228029727935791, -0.3396650552749634, 1.239604115486145, 0.18893559277057648, 0.4336981177330017, -0.7471460700035095, -0.6286888718605042, 0.32941243052482605, -0.7959111332893372, 0.7072275280952454, -0.22246205806732178, -1.262760043144226, -0.2353571355342865, 1.0604020357131958, 0.03791724517941475, -1.1894934177398682, 1.3700991868972778, -0.44453859329223633, -0.08806634694337845, -1.5790362358093262, 0.5258254408836365, 0.07569685578346252, -0.2960048019886017, -0.24909868836402893, 0.004699283745139837, 0.5783076882362366, -1.1673057079315186, -1.876577377319336, -1.807911992073059, -1.2159367799758911, 0.588508129119873, -1.202632188796997, 1.0063652992248535, -0.2040041983127594, 0.343589723110199, -0.9259090423583984, -1.460976243019104, 0.09096646308898926, -0.17650219798088074, 0.5689712166786194, 0.03747369349002838, -0.37773796916007996, -1.901213526725769, -0.24825987219810486, -0.6744221448898315, -1.2762269973754883, 1.1107097864151, 0.13142210245132446, 1.9491400718688965, -0.9472928643226624, -0.6472570896148682, -0.7647805213928223, -0.01435520313680172, 1.3807036876678467, -1.2276008129119873, 0.8732187151908875, 0.7670314311981201, 0.10818132758140564, 1.0303540229797363, 0.7328693866729736, 0.801215410232544, -1.3260794878005981, -2.0389721393585205, -0.4360640048980713, 0.18087050318717957, -0.8091251254081726, 1.2342208623886108, -0.7273780107498169, 0.7121583819389343, -0.20030316710472107, -0.7488389611244202, -0.14250792562961578, 2.0939853191375732, -0.7774232029914856, 0.2599734365940094, 1.9471246004104614, 0.22012485563755035, 0.21485471725463867, 0.08449095487594604, -0.2973504066467285, 0.6333425045013428, 2.0126752853393555, 0.28932473063468933, -0.96904057264328, -0.5522581338882446, -0.755142867565155, -0.021740378811955452, -0.8270692825317383, 1.0247708559036255, -0.5383602380752563, 0.48385778069496155, 0.8168537616729736, 0.24516458809375763, 1.3458044528961182, 1.4875575304031372, -0.8090881109237671, -0.7830881476402283, 0.36422625184059143, 0.6267188191413879, -0.5586491227149963, 1.1677427291870117, -2.033082962036133, 0.017064806073904037, -1.054413080215454, 0.03317252919077873, 1.9619567394256592, -0.6160150170326233, -1.4079281091690063, 0.36810600757598877, -0.8750783205032349, 0.6616291403770447, -0.34385427832603455, -0.4416554570198059, -0.62919020652771, 0.036031272262334824, 1.1899389028549194, -0.5364791750907898, 0.04681473970413208, 0.6858339905738831, -0.46117866039276123, -0.5597935914993286, 0.08454573899507523, 0.7921103239059448, -1.5975866317749023, -0.7611154913902283, -0.9914212822914124, -1.9107016324996948, -0.8074570894241333, -1.3822879791259766, -0.11598413437604904, 0.14927193522453308, 1.44209885597229, -0.3386991024017334, 3.538999080657959, 1.3269169330596924, -0.08505401015281677, 0.10466016083955765, -0.3020952343940735, -0.9689255952835083, 0.42107686400413513, -0.44235655665397644, -1.9969843626022339, -1.0808266401290894, -2.420691728591919, -0.838196337223053, 0.32473692297935486, -0.5924844741821289, -0.5154112577438354, 0.4756753146648407, 1.0409135818481445, -0.59504634141922, 1.0368448495864868, 0.07572633773088455, 0.3315724730491638, 1.2363406419754028, -0.2900193929672241, 0.8188738822937012, -0.149723619222641, 0.3233937621116638, 1.2106246948242188, -0.9534186124801636, 2.2626945972442627, 0.3023795187473297, 1.4083738327026367, 0.09831568598747253, 0.4125494360923767, -0.5781263709068298, 2.0032927989959717, 1.6636457443237305, -0.46081361174583435, -1.066084861755371, -0.7545512318611145, 0.9182125926017761, 0.6028773784637451, -0.6607245206832886, -0.28502437472343445, -0.19400310516357422, 0.038008224219083786, -0.6255795359611511, 0.41388046741485596, -1.9044756889343262, -0.5230667591094971, -1.7812687158584595, 1.295059323310852, -0.4530017375946045, -0.39891037344932556, 0.46760624647140503, -0.1952822059392929, 1.4959869384765625, -0.23200547695159912, 2.358416795730591, 0.8599700331687927, 0.6487664580345154, -0.15009304881095886, -0.5790250897407532, -0.5576860308647156, -0.7650452852249146, -1.369742751121521, -2.1807916164398193, 0.4577482342720032, -0.427243709564209, -0.906583309173584, -0.4665319323539734, 1.2197301387786865, 2.377331256866455, -0.15449415147304535, 0.23954717814922333, 0.8837125897407532, 1.0376782417297363, -0.07048507034778595, -0.48313552141189575, 2.8293309211730957, -0.48810479044914246, -0.9576032757759094, 0.7122403979301453, 0.002438061172142625, -0.7254663109779358, -0.4110439121723175, 0.6546944379806519, 0.15422925353050232, -0.776114821434021, -0.7899681925773621, -0.7377893328666687, -0.8087996244430542, -0.7472504377365112, 2.0250134468078613, -0.05351540073752403, 0.11525082588195801, -1.2913811206817627, -0.12553758919239044, -1.5920028686523438, 0.9367871284484863, 1.9187055826187134, -0.06273412704467773, 0.041020315140485764, -0.14875894784927368, -0.10839784145355225, -0.003759808372706175, 1.003352165222168, -1.1149400472640991, 1.3023256063461304, -1.1878509521484375, 0.6076620221138, -2.3272299766540527, 1.8412412405014038, -1.4419382810592651, 1.1753703355789185, -2.1890461444854736, -0.9657126069068909, -2.0540034770965576, -0.37035906314849854, -0.7947258353233337, -0.6909308433532715, 1.1919571161270142, 0.41166970133781433, 0.883834719657898, -0.32883724570274353, 0.37914279103279114, -0.7794554829597473, -2.842776298522949, 1.121443510055542, 0.7406986355781555, 0.6339856386184692, -0.256530225276947, 0.19202780723571777, 1.0285956859588623, -0.17956086993217468, -0.44327354431152344, -0.5598064661026001, 0.7291990518569946, 0.7319157719612122, -0.14748422801494598, -0.7493231296539307, 1.3401836156845093, 1.0786622762680054, 0.12797784805297852, -0.780697226524353, -1.7055294513702393, 1.3289567232131958, -0.9029543995857239, -0.9789481163024902, -0.2576659321784973, 0.37942445278167725, 0.4186316728591919, -0.24369806051254272, 1.6341829299926758, 0.40572401881217957, -0.8047869801521301, -0.9952383637428284, 0.3969670236110687, 0.45419713854789734, -0.283792644739151, -2.488166570663452, -0.4575441777706146, -0.8361119627952576, -0.332694947719574, 0.6600301265716553, 0.07844389975070953, -0.014013189822435379, -0.07702433317899704, 1.6386218070983887, -1.2618006467819214, 0.07722551375627518, -1.754380464553833, -0.6835442781448364, -0.7258908152580261, -0.8173474073410034, 0.29514285922050476, 1.288862705230713, 0.8052154183387756, -0.39062434434890747, 2.0656557083129883, 0.5759751796722412, -0.034806713461875916, 0.5357770919799805, 0.2813735902309418, 0.07095159590244293, 0.8560061454772949, 0.043008528649806976, -0.41916605830192566, -2.09820294380188, 2.1149322986602783, -0.06780614703893661, -0.8396346569061279, -1.3228839635849, -0.7528219223022461, 0.05356943607330322, -1.0582412481307983, -0.6399311423301697, -0.6250866651535034, 0.09562018513679504, 0.8150312900543213, 0.7381241917610168, -0.8244691491127014, -1.746172547340393, 1.170981764793396, -0.027748821303248405, 0.8312501311302185, 1.1862924098968506, -0.08689441531896591, -1.4058634042739868, 0.4561312198638916, 1.6890428066253662, -0.13615292310714722, 0.13606393337249756, -0.9502964615821838, -0.1750139445066452, -0.3897600769996643, -0.6327406764030457, 0.8203811645507812, -0.5269773006439209, -0.5666565299034119, -1.3020657300949097, 1.631418228149414, 0.7372207045555115, 0.1048830971121788, 0.8466534614562988, -0.5598227977752686, 0.36731356382369995, -0.056566040962934494, 0.30740949511528015, -0.09266415238380432, -0.7979373335838318, 0.9737534523010254, -2.1781632900238037, 0.25406691431999207, 0.648677408695221, 0.4686316251754761, -0.02767869085073471, -0.23330938816070557, 0.9630876183509827, 0.5619730949401855, 1.5828427076339722, 0.015368545427918434, 1.5211358070373535, -0.012797662988305092, 0.5850170254707336, -0.7918634414672852, -0.41261497139930725, 1.0115482807159424, -0.7537541389465332, 0.11945797502994537, 0.2437441051006317, -1.7092679738998413, -0.16431047022342682, -0.9656515121459961, 2.130648612976074, -0.42028966546058655, -0.5311089158058167, 1.2769829034805298, -0.17771311104297638, 1.0293171405792236, -0.040595293045043945, -0.09015262126922607, -0.5210121273994446, 0.4770258367061615, 1.5049660205841064, 1.5202440023422241, -0.07414646446704865, -1.1771306991577148, -1.376045823097229, -0.2729092240333557, 0.9081052541732788, -0.631107747554779, 0.805682361125946, -1.8549922704696655, 1.299312710762024, -0.9872516393661499, -0.4299474358558655, -1.4388657808303833, -0.6111461520195007, -1.6317082643508911, -0.2404099404811859, 1.2860571146011353, -0.44136378169059753, -1.1435716152191162, -0.738003671169281, -0.8136922121047974, 0.30227750539779663, 0.27538758516311646, 0.6598092913627625, -1.2413015365600586, 1.4793974161148071, -1.640417456626892, 0.25408902764320374, -0.34290066361427307, -0.608657717704773, -0.06732559204101562, 0.9097155332565308, 0.9114356637001038, 0.44910386204719543, -0.0444723516702652, -0.5949447154998779, 1.0364634990692139, -0.09741947054862976, 0.7754272818565369, -0.5896947383880615, -1.0849295854568481, 0.44716984033584595, 1.599829912185669, 0.7547054886817932, -0.20122139155864716, -0.9744700193405151, 0.27234140038490295, 1.208114743232727, 1.2931517362594604, -0.5764468312263489, -1.1334642171859741, -0.4689437747001648, 0.051947373896837234, 0.04663556069135666, -0.6915518641471863, -1.1812063455581665, -0.11112029105424881, 0.1245485320687294, 2.6476240158081055, -1.5995854139328003, -0.05540444701910019, -0.35603150725364685, -1.0775049924850464, 0.049704037606716156, -0.8161195516586304, -0.4491681754589081, 0.3822605013847351, 1.4224385023117065, 1.137833833694458, -1.137044072151184, -1.344351887702942, -0.23240680992603302, -0.20448806881904602, 0.07859059423208237, -0.46175554394721985, 0.008131945505738258, -0.4326011538505554, -0.988501787185669, 0.5856835246086121, -1.8253229856491089, -1.0793696641921997, -0.008130581118166447, 0.5174599289894104, -1.1903250217437744, 0.13166823983192444, -1.2243388891220093, -0.5833777189254761, -0.9473978877067566, -0.5893709063529968, -1.6506789922714233, -0.6699042320251465, -0.48073750734329224, 0.09204448014497757, 0.7357388138771057, -0.3353380262851715, 1.7438887357711792, 0.700573205947876, 0.6214438080787659, 0.33815595507621765, -1.3910595178604126, -1.93569016456604, 0.23778678476810455, -0.07867466658353806, 0.3653007745742798, -0.5363856554031372, 0.07261548936367035, 0.7048155069351196, -1.0209983587265015, 1.6499018669128418, 0.824888288974762, 0.4279705286026001, -0.16188883781433105, 1.700319528579712, 1.1531153917312622, -0.4536668062210083, -1.0817326307296753, 0.9704821109771729, 1.8682750463485718, 0.6858725547790527, -1.1304301023483276, -0.44554412364959717, -1.6088248491287231, -1.055910348892212, 0.7768641114234924, 0.2164410501718521, -1.9626256227493286, 0.20192690193653107, 0.7263807654380798, 0.5044887065887451, -0.07336510717868805, 0.37032240629196167, 0.862673282623291, 1.5628809928894043, 0.8304898738861084, 0.6357212066650391, -0.8222858309745789, -1.1988801956176758, -1.7600464820861816, 0.13158535957336426, -0.8504550457000732, 0.25907793641090393, -1.2894668579101562, 0.3098820745944977, 1.283115267753601, -0.4069238305091858, 1.8510698080062866, -1.404341697692871, -0.8119187355041504, -2.998232126235962, 2.4420175552368164, 0.7805322408676147, -1.0399646759033203, 2.61037278175354, 0.14555644989013672, 0.35877740383148193, -0.041851866990327835, -0.9559496641159058, 0.7869643568992615, -1.4661555290222168, -1.9451841115951538, -0.01855889894068241, 0.6450899243354797, -0.4625953435897827, 0.628213107585907, -0.36406293511390686, -1.6521402597427368, -0.14907798171043396, 0.25193434953689575, -1.0338138341903687, 0.8472322225570679, 1.2891199588775635, -0.13628701865673065, 0.9843500256538391, -1.113614559173584, 0.17220191657543182, -0.8846228122711182, -1.1389988660812378, -0.09357734769582748, 2.2070956230163574, 0.20027939975261688, -1.2143276929855347, -0.4718494713306427, -1.0236787796020508, 0.9987456202507019, -2.121107578277588, -1.1015180349349976, -0.0005493528442457318, -0.08875449746847153, 0.019641024991869926, 1.0749363899230957, 0.5179095268249512, -1.733553171157837, -2.774566650390625, -0.31641727685928345, -0.48309090733528137, 1.031425952911377, 0.5695306658744812, -0.8140177130699158, -1.093611717224121, -0.5927272439002991, -0.714364767074585, -1.761934757232666, -1.2756621837615967, -1.0017329454421997, -1.4951237440109253, -0.3550401031970978, -0.16456182301044464, 0.011533240787684917, 0.47990575432777405, -0.9559430480003357, 0.8813530802726746, 1.9244998693466187, -0.06740153580904007, -0.5285184383392334, 0.36520838737487793, 0.28496068716049194, -0.9709746241569519, -0.022677702829241753, -0.18187382817268372, -0.36807113885879517, -0.7621496319770813, 0.24194689095020294, 1.5164045095443726, 0.921836793422699, -0.638989269733429, -2.076669692993164, 0.17566990852355957, -0.21010132133960724, 0.26377928256988525, 0.7030872702598572, 0.3266304135322571, -0.07522612810134888, 1.2741425037384033, 0.053252022713422775, -0.24669484794139862, 1.2764116525650024, 0.8420705795288086, 0.07156404852867126, -0.5529718399047852, 0.7156832814216614, -2.253948211669922, 0.3989454209804535, -1.553505778312683, -1.4291847944259644, 1.506299376487732, 0.991309642791748, -0.6550793051719666, 0.13750167191028595, -1.0854127407073975, -0.35649460554122925, -0.028473757207393646, -0.806001603603363, -2.196654796600342, -1.333655834197998, -1.238434910774231, -0.1985878199338913, 0.29870879650115967, -0.23715579509735107, 1.0965158939361572, -1.7872123718261719, -2.6913106441497803, -1.0372321605682373, 0.1647048145532608, 0.5613018870353699, -3.032637357711792, 0.8545994758605957, 0.5217417478561401, 1.0716133117675781, 0.07093066722154617, -1.2782232761383057, 0.8127145171165466, 0.27731773257255554, -0.2236030399799347, -1.0443974733352661, 0.39465489983558655, 2.209433078765869, -2.3915810585021973, -0.11067922413349152, 0.013999361544847488, -0.5800871849060059, 0.2263542264699936, -0.6044990420341492, 1.0673660039901733, -1.084883213043213, -0.5723085403442383, -0.42197293043136597, -1.2829087972640991, 0.36373379826545715, -1.2065843343734741, 0.4763733744621277, -1.099472999572754, 0.327345073223114, -0.4494786262512207, 0.6044625043869019, 0.031736984848976135, -1.3579682111740112, -1.315313696861267, -0.3714907467365265, -0.9121865034103394, 1.8287526369094849, -0.5658149123191833, 0.39598846435546875, 0.5428400039672852, 0.05343896523118019, 0.1097046285867691, -1.475046992301941, -0.9118505716323853, 0.2172228842973709, 1.2793105840682983, -0.26164311170578003, -1.307317852973938, 0.783750057220459, 0.41177332401275635, 2.546114206314087, -1.5779004096984863, 1.2634730339050293, -1.0335421562194824, 1.2719663381576538, 1.4423940181732178, 1.5848488807678223, 0.47597119212150574, -1.757970929145813, 0.9544618725776672, -1.5620263814926147, -1.06183922290802, -0.6701148152351379, 0.21967341005802155, -0.049597106873989105, 0.04904299974441528, 0.6006343364715576, 0.09458070993423462, -0.9268895983695984, -0.3801950216293335, -1.3945447206497192, 1.5575828552246094, 0.8657228350639343, -1.0772649049758911, -0.9741163849830627, 0.411880761384964, -0.2974272072315216, -0.8038892149925232, 0.9616995453834534, -1.3300281763076782, -0.2716834545135498, 0.4245685040950775, 1.5697089433670044, -0.3845021724700928, -1.2620251178741455, 0.6310333609580994, -0.604288637638092, -0.5972777009010315, 0.15789620578289032, -0.706203818321228, -0.546467125415802, 0.6466041803359985, -2.4092373847961426, -1.8357319831848145, -1.370215654373169, -1.0854992866516113, 0.40327274799346924, -1.3090102672576904, -2.121497869491577, 1.011841058731079], [0.12312249094247818, 0.4951034188270569, 1.479516625404358, 0.2525198757648468, -0.745354175567627, 1.0617338418960571, -0.0652257576584816, -0.5343349575996399, -0.5728638172149658, 0.45388543605804443, 1.598152756690979, 0.5009938478469849, -0.7868493795394897, 1.3593584299087524, 0.7221015691757202, -1.3750816583633423, 0.3722348213195801, -1.0809050798416138, 0.10853780061006546, -0.025812050327658653, 0.09378424286842346, 1.2001709938049316, -0.4545929729938507, 2.061824083328247, -0.22149336338043213, -1.8515163660049438, 1.8249220848083496, -0.7071525454521179, 0.14079409837722778, 1.5427595376968384, 0.16362331807613373, 0.07508913427591324, 0.7184175848960876, 0.12967602908611298, 1.4717962741851807, 0.4396495223045349, 0.3390671908855438, -0.9181249737739563, -0.3197847306728363, -0.1561397761106491, -3.1646571159362793, 0.42950573563575745, -2.2624099254608154, -0.9527813792228699, -2.7297489643096924, -0.7617625594139099, 1.2847174406051636, 0.3722544312477112, -0.04240172728896141, 0.1926366239786148, 0.4942765533924103, -1.167046308517456, 0.8836256861686707, -0.24805179238319397, 1.6942927837371826, 0.7883113622665405, -0.2720271944999695, -0.6128944158554077, 1.201842188835144, -0.4601882994174957, -0.4322132468223572, -0.8555318713188171, -0.4110592305660248, 0.01128925010561943, -0.02991948276758194, -1.275736689567566, 0.06501486897468567, -0.3637215793132782, 0.5478028655052185, -1.1012545824050903, -0.5564973950386047, 0.590002179145813, 0.8929290175437927, 0.4752916991710663, -0.9686000943183899, 0.34774863719940186, -1.425784707069397, -0.20512248575687408, -0.9917756915092468, -0.08309721201658249, -0.4116419553756714, -1.046538233757019, 0.0541844516992569, 1.1718502044677734, -1.3818331956863403, -1.559259057044983, -1.4114952087402344, -0.1906503587961197, 0.7534526586532593, 0.24754256010055542, 2.1818718910217285, 1.2241175174713135, 0.33984172344207764, 0.3357471227645874, 1.2731592655181885, 0.9302467703819275, -1.6404422521591187, -0.898732602596283, 0.04986618086695671, 0.7915453910827637, 0.21892127394676208, -0.2490645945072174, -1.6586103439331055, 1.0809862613677979, 0.3084326684474945, 0.505247950553894, -0.18528109788894653, -1.115825891494751, 0.4406639635562897, -0.716691792011261, 1.617599606513977, 0.44909220933914185, -0.3386302590370178, 2.977543354034424, -1.7611526250839233, 0.6587305665016174, 1.294463038444519, 0.7100952863693237, 0.2527172863483429, 0.5228155851364136, 2.0128870010375977, 2.541454315185547, -0.4266775846481323, -1.1533890962600708, -0.31029918789863586, 0.9361079335212708, -0.3907116949558258, -0.407626211643219, 0.34201034903526306, -0.3963252604007721, -0.7344699501991272, 2.4656360149383545, -0.7888525128364563, -1.4381048679351807, 0.04861810430884361, -0.4806363582611084, 0.5744272470474243, -1.8690710067749023, 1.9159349203109741, 0.5272212028503418, 1.5314961671829224, -0.6657519936561584, 0.4316572844982147, -2.3283140659332275, 0.20531272888183594, -1.309977412223816, -0.5375391840934753, 1.5200673341751099, 0.9504338502883911, 0.6743550896644592, 0.5984786748886108, 1.1497366428375244, -0.4099237620830536, -0.44378775358200073, 0.7164252400398254, -2.7432901859283447, 0.20210009813308716, -0.09159591048955917, 0.4846619963645935, -0.1360970139503479, 0.9884909391403198, 0.20864470303058624, -0.26373130083084106, -0.7229488492012024, -0.15214896202087402, -0.8939397931098938, -1.1533422470092773, 0.016863249242305756, -0.3477601706981659, 0.07353703677654266, -1.3860408067703247, -0.8390860557556152, 1.319055438041687, -2.432323694229126, 1.842612862586975, 0.11437975615262985, 0.0608731247484684, -0.3246435225009918, -1.3559120893478394, -1.1502498388290405, -0.20164678990840912, -0.7422166466712952, -0.0898323580622673, -1.4783562421798706, -0.1838638186454773, -0.11437762528657913, -2.8009469509124756, -1.3140358924865723, -0.7201934456825256, -0.5401378870010376, 0.4567645788192749, 1.290429711341858, 1.0334337949752808, 0.4569106698036194, 0.4790174663066864, -1.5217742919921875, 0.05000904202461243, -1.350587248802185, -0.6206707954406738, 0.19448241591453552, -1.5073782205581665, -0.5286034345626831, 1.5566614866256714, 0.6286560297012329, -0.4397580921649933, -0.2975408434867859, -1.3131550550460815, -0.14943751692771912, -1.4069592952728271, -1.378708004951477, -1.1224524974822998, 0.5375648736953735, 1.010873556137085, 0.11580316722393036, -0.5010159015655518, 0.005403326824307442, 0.08584577590227127, -0.7035354971885681, 2.3023056983947754, -0.6146928071975708, -0.013570393435657024, -0.16844694316387177, 0.9399927854537964, 1.2375680208206177, 0.3009868264198303, 1.5346177816390991, 0.29511889815330505, 0.6689611077308655, 0.008253510110080242, 1.112328052520752, 0.05619930848479271, 0.28949251770973206, 1.2806859016418457, 1.5514905452728271, -1.426003098487854, -0.7330687642097473, -1.746690034866333, -0.9316437244415283, -0.581416130065918, 0.37430763244628906, 2.9458651542663574, -0.6333726644515991, 1.8980156183242798, 0.3025636672973633, -0.15817153453826904, 1.4171005487442017, -0.7760943174362183, -0.6021106243133545, -1.1204683780670166, 2.0937247276306152, -0.4502110779285431, -0.6655656099319458, -0.8921704292297363, 0.6095555424690247, 0.06994806975126266, 1.3257768154144287, 0.12213027477264404, -0.13120369613170624, -0.8974362015724182, 0.07147736847400665, -0.805055558681488, -0.44388338923454285, -0.42766478657722473, 0.9697884917259216, 0.6321071982383728, -0.4349571168422699, 0.940112292766571, -0.15361051261425018, 0.7170460224151611, 0.9364385604858398, -1.1873537302017212, 0.7935457229614258, 1.9623196125030518, 1.0919890403747559, 0.8679977655410767, -0.6202774047851562, 0.10155512392520905, -0.06915096938610077, 0.4800618290901184, -1.6828758716583252, -0.2964431047439575, -0.5866603255271912, -1.5039106607437134, -1.2726680040359497, -0.1507667601108551, -0.11420733481645584, 1.9400229454040527, 0.7367798686027527, 0.6368368268013, -0.012268771417438984, 1.4399901628494263, -0.08019054681062698, 0.916661262512207, 1.1399158239364624, 0.5033602118492126, 0.18226021528244019, 0.04020727425813675, 0.6453201174736023, 1.3599607944488525, -0.39257705211639404, -0.47221675515174866, 1.1694172620773315, -0.6640627980232239, 0.33932381868362427, 1.2287967205047607, 0.7740718126296997, 0.16958272457122803, 0.31620126962661743, 1.4394514560699463, 0.35057753324508667, -0.484112948179245, -0.7308377623558044, 1.5205358266830444, -1.1586834192276, -0.423546701669693, 2.3420398235321045, 0.37560731172561646, -1.7848514318466187, 0.688818097114563, -0.9055863618850708, -0.06887838244438171, -0.10173068195581436, -0.9453298449516296, -0.009729281067848206, 0.3290599584579468, 1.2224787473678589, -1.6681064367294312, 0.4094461500644684, 1.0425506830215454, 0.5929576754570007, -0.44732797145843506, -1.2541015148162842, -1.604275107383728, -0.6775322556495667, 0.7580550312995911, -1.4238646030426025, 0.40697982907295227, -0.20725370943546295, 0.18893305957317352, 1.1373811960220337, -0.12181971222162247, 1.236194133758545, -0.19183699786663055, -0.11542097479104996, -0.2766093313694, 0.1785350888967514, -0.8975849151611328, -1.0396138429641724, -0.33977624773979187, -0.6831010580062866, 1.538556694984436, -0.2914981544017792, -0.06769529730081558, -0.32568296790122986, -0.2927842438220978, -0.23549281060695648, 0.20477727055549622, 2.029343366622925, -0.20099258422851562, -1.1640563011169434, -0.806656002998352, 0.7641645669937134, -0.357174277305603, 2.346177577972412, -1.4392335414886475, 1.4202395677566528, -1.1474663019180298, 0.7458592653274536, 2.2786903381347656, 0.7602518796920776, 0.04710248112678528, 1.045072078704834, -0.5948411226272583, 0.06398399919271469, -0.028958410024642944, -1.6205110549926758, -0.09447465091943741, -0.35875409841537476, 1.6474641561508179, -0.6060062646865845, 0.5556573271751404, -0.09093371778726578, -1.2360329627990723, 2.591038227081299, -0.08813662827014923, -2.0642216205596924, 0.7635968923568726, -0.8642568588256836, -1.30301034450531, -0.6782047152519226, -0.33130714297294617, 0.2602125406265259, 1.8129299879074097, -0.9354467391967773, -1.406459927558899, -1.5710034370422363, -1.4660091400146484, -1.121442198753357, -0.3504822552204132, -1.6161881685256958, -0.9784107208251953, -0.9918317198753357, -0.9275243878364563, -0.7127023339271545, 0.5097190141677856, 1.0699888467788696, -1.730881690979004, -0.1633361577987671, 2.463292360305786, 0.09456729143857956, -1.6938680410385132, 0.04918956011533737, -0.45575347542762756, -0.3367041051387787, -1.1464972496032715, 0.20679053664207458, -0.1119120791554451, -0.8404441475868225, 0.41554197669029236, -0.09508862346410751, 1.4518002271652222, -0.6216752529144287, 2.176771402359009, -0.6609257459640503, -0.8809554576873779, 0.3356945812702179, 0.37411072850227356, 0.6576050519943237, 0.2625046968460083, 0.5935991406440735, -0.9749054312705994, -0.09075579047203064, 2.0448553562164307, 1.1816329956054688, -0.2239285111427307, 0.19003862142562866, 0.8665764331817627, 0.4333898425102234, -1.0272055864334106, -0.26689985394477844, -1.5719938278198242, 1.6997636556625366, -1.374550700187683, -0.13603158295154572, 0.23380738496780396, 0.07313895225524902, -0.1546197235584259, 0.8341786861419678, -0.8385947346687317, -0.0885096862912178, 1.0913567543029785, -1.616073489189148, -0.41842567920684814, 1.4522819519042969, -1.434837818145752, -0.4814969599246979, -1.3440921306610107, -2.342728853225708, -0.49244531989097595, -0.9738492965698242, -1.561832308769226, -1.3791780471801758, 0.598883330821991, 0.4588288962841034, -0.6861342787742615, -1.2503629922866821, -0.9889547824859619, -0.6861473321914673, -0.19222396612167358, -0.20747432112693787, -0.4608278274536133, -0.10647603124380112, 1.0357401371002197, 0.6155234575271606, -0.00021916741388849914, -0.8839332461357117, -1.890812635421753, 1.1671065092086792, 0.8112233281135559, -0.30851563811302185, 0.33688512444496155, 0.8205356597900391, 0.27187684178352356, 0.9430103898048401, -1.1466656923294067, -0.4323582351207733, -0.21036295592784882, -1.3984993696212769, 0.4040065407752991, 1.1189926862716675, -0.3202243149280548, -0.5169737339019775, -0.3425684869289398, 1.0245119333267212, 0.9532578587532043, 0.15531158447265625, -0.35983777046203613, 1.2505816221237183, 0.09736141562461853, -0.6087676882743835, -0.4176366627216339, -0.32803526520729065, -0.014450748451054096, -0.6058977842330933, 1.031265139579773, -0.19696009159088135, 0.31765881180763245, -0.08983035385608673, -0.9944100379943848, 0.13420790433883667, 0.19007715582847595, 1.6572380065917969, 0.2456417679786682, 1.914238452911377, -0.1249522790312767, 0.4563415050506592, 1.9672945737838745, -0.07009872049093246, -0.46951794624328613, 0.40055060386657715, -1.4825420379638672, -1.4318511486053467, -1.0170639753341675, -2.8389647006988525, 0.6375096440315247, -0.848649799823761, -0.27301695942878723, -0.6541640162467957, 0.5450147986412048, -0.5551188588142395, 0.8273184299468994, -2.082751750946045, 1.7487514019012451, 0.17664872109889984, 0.035605140030384064, 0.24309931695461273, -1.9404586553573608, -1.1690113544464111, -0.5358400940895081, 0.24318642914295197, 1.1206800937652588, -1.630953073501587, -1.7821205854415894, 1.1693143844604492, -1.240063190460205, -0.2936977446079254, 1.0100189447402954, -1.5865665674209595, -0.987494707107544, 0.14433982968330383, 0.5014339685440063, 0.2946567237377167, -1.4911439418792725, -1.1929831504821777, -1.9937937259674072, -0.3606933355331421, 0.0780617743730545, 0.3613598048686981, -0.5486091375350952, -0.25924405455589294, -0.6413999199867249, -0.9710080027580261, 1.662949562072754, -0.8571372628211975, -2.3119490146636963, -1.8310878276824951, 0.30271872878074646, 0.4211009442806244, -0.5816583037376404, 0.8558003902435303, 1.2060598134994507, 0.792665421962738, 0.2443121373653412, -1.3506016731262207, 0.5881903767585754, 0.43455711007118225, -1.288596749305725, 1.1485612392425537, 1.0771201848983765, 0.6964142322540283, -0.9797929525375366, 0.15140774846076965, -0.30237889289855957, 3.210012674331665, -2.043416976928711, -1.3437167406082153, 1.2515519857406616, -0.3388809263706207, -0.3429790735244751, 0.09173174202442169, -1.2640933990478516, -1.0786014795303345, -0.6058840155601501, -0.040364693850278854, -1.604994297027588, -0.12952278554439545, -0.7722142338752747, 0.6400830149650574, -0.9373599886894226, 3.1675186157226562, -1.3648512363433838, 0.8355451226234436, 1.5788010358810425, 1.6958324909210205, -2.3702802658081055, -0.42932090163230896, 0.297435998916626, -0.23232102394104004, 1.5065141916275024, 0.31259578466415405, 0.20462021231651306, 1.3475669622421265, 0.6363436579704285, 1.7479450702667236, 0.6141681671142578, -0.6944547295570374, 0.6295130848884583, 1.091543436050415, -1.1103371381759644, 0.8681720495223999, 2.178741931915283, -0.4717170000076294, -0.7853766679763794, -0.8373922109603882, 1.0493977069854736, 0.38086003065109253, 0.7390336990356445, 0.7346517443656921, 0.6324576735496521, -0.62960284948349, -2.3223745822906494, 0.5886677503585815, 0.7009636163711548, 2.8896665573120117, 1.68381667137146, -0.39804354310035706, 1.949629783630371, 0.35670575499534607, 0.7529711127281189, 3.79038405418396, -1.3243496417999268, 1.0406819581985474, 1.1640400886535645, 0.02985721081495285, -0.3924730718135834, -0.19338098168373108, 0.8390985727310181, 0.8564572930335999, 1.2645354270935059, -1.164081335067749, 0.4064192771911621, 0.5426740050315857, 2.14094614982605, 0.8694033622741699, 1.2481228113174438, 0.1355794072151184, 0.42445433139801025, -0.10203273594379425, -0.42156660556793213, -0.5246841311454773, 0.09167002886533737, 0.6312277913093567, 0.6821199655532837, 1.0493568181991577, -1.115476369857788, -2.0430705547332764, 0.5615046620368958, -1.3419685363769531, -0.513805627822876, -1.078605055809021, 0.36064615845680237, -0.17825321853160858, 0.292951375246048, -1.3637615442276, -1.5438764095306396, -0.07721294462680817, -0.5261793732643127, -0.27626267075538635, -1.7443643808364868, 1.389359712600708, 2.5261595249176025, 0.8919273018836975, -0.6440166234970093, -0.544910192489624, 0.7401788830757141, 0.609245240688324, -1.0725435018539429, 1.0244899988174438, 0.1291535198688507, 0.057716004550457, -0.29668039083480835, -1.6317551136016846, 2.013502836227417, -0.08728653937578201, 0.2442736178636551, -2.1793673038482666, -0.44605353474617004, 0.03552841395139694, 1.7228182554244995, -0.7140143513679504, 1.247071623802185, -0.424528568983078, -0.9054335951805115, 0.5231202244758606, -0.605797529220581, 1.9379768371582031, 0.5381855368614197, -0.5866017937660217, 2.1892781257629395, -1.5790964365005493, -1.165684461593628, 2.2993900775909424, -0.3344939947128296, 0.28028222918510437, -0.6913692355155945, 0.6465495228767395, -0.19285759329795837, 0.6608796119689941, 1.3057576417922974, -0.28340455889701843, -0.603323757648468, 0.5448766350746155, 0.4416588544845581, 1.0872949361801147, -0.3394016623497009, 1.211625337600708, -0.1708517223596573, 0.41045331954956055, 1.010506510734558, -2.9964823722839355, -2.7029898166656494, 0.9336493015289307, 0.11230635643005371, 0.44452622532844543, -0.5185431241989136, 0.5950372815132141, 1.7937335968017578, 1.2016972303390503, -0.3297167122364044, 1.1601753234863281, -1.6468247175216675, -1.301732063293457, -1.1852686405181885, 1.5900311470031738, 1.4720102548599243, -1.9772796630859375, 0.7656961679458618, 0.34985116124153137, 1.0064588785171509, -1.0671441555023193, 1.567674994468689, -1.4730803966522217, -0.020392360165715218, 0.3999023735523224, -0.6454153656959534, 0.28399357199668884, 0.6017008423805237, -0.4981648325920105, -0.28450947999954224, -1.3430346250534058, -1.3882445096969604, -0.35436850786209106, -1.4164632558822632, -1.1336827278137207, 2.0197298526763916, 0.5348823666572571, -0.8807758092880249, -1.5759036540985107, 1.2991528511047363, -0.16808269917964935, -1.1072285175323486, 0.997046709060669, 2.8335378170013428, 0.19400109350681305, 2.4502320289611816, -0.4149911105632782, 0.5223748087882996, 0.8034029603004456, 0.41305410861968994, 1.8261117935180664, 0.32201120257377625, -2.2759571075439453, 0.14955194294452667, 0.14527203142642975, 0.3265039920806885, 0.38613560795783997, -0.32533228397369385, -0.5761283040046692, 0.20019523799419403, 0.8268409967422485, -0.18762385845184326, 0.17381440103054047, -0.7639807462692261, 2.0831916332244873, 0.4010409712791443, 0.8389117121696472, -0.7901478409767151, 0.447230726480484, 0.37944111227989197, -0.20051094889640808, 0.22359596192836761, -0.20040613412857056, 0.7973084449768066, -0.4777578115463257, 1.3865896463394165, -0.08759742975234985, -1.300256371498108, 0.24615390598773956, -0.48771730065345764, 1.7397003173828125, 1.146574854850769, 0.9712688326835632, -0.39462706446647644, 0.8778640627861023, -0.10126367956399918, -0.12282481044530869, 0.7434911727905273, -0.18827252089977264, 1.4494224786758423, -1.8388116359710693, 1.9438339471817017, 1.7256975173950195, -0.6188656091690063, 0.8487529754638672, -0.3368252217769623, 0.4930761158466339, 0.5596404671669006, 0.21201074123382568, 0.48270153999328613, -0.9568809866905212, 1.2550101280212402, 0.28211987018585205, -0.05144713446497917, -0.551840603351593, -0.04187922552227974, 1.8364014625549316, 0.24796122312545776, 0.5580837726593018, 0.6579141020774841, -0.12216778844594955, 2.030519962310791, 1.2867991924285889, 0.8031359910964966, 0.07400597631931305, 0.0624973326921463, -0.5303173661231995, 0.6905115246772766, -1.8699462413787842, 0.45540496706962585, -0.568700909614563, 0.4798150658607483, 0.8150719404220581, -1.114554524421692, -0.9778683185577393, 1.1340688467025757, -0.41314664483070374, -0.21553924679756165, 0.28637638688087463, 0.414980411529541, -0.2473016083240509, -0.3548557758331299, -0.6453275084495544, 0.26736268401145935, 2.107745409011841, -0.30376917123794556, -0.7653175592422485, -0.20278626680374146, 0.4868636131286621, 0.5075092315673828, 0.9839938282966614, 1.5757352113723755, 0.7671376466751099, 0.20783580839633942, 0.5998879671096802, 0.7723511457443237, -0.9588505625724792, 0.1133873239159584, -0.8534975051879883, -0.5236765146255493, -0.5851873159408569, 0.24883678555488586, 1.660664439201355, -0.7405920028686523, -1.391320824623108, 0.11171415448188782, 0.326206773519516, 0.5564137101173401, 0.428120493888855, -0.35414090752601624, 0.5689276456832886, -0.5722473859786987, -0.568084716796875, 1.1177300214767456, 0.8540358543395996, -1.1142973899841309, 0.787930965423584, 0.4044620096683502, 0.8502804040908813, 0.5321195125579834, 0.39325448870658875, 0.08439783751964569, -0.32056036591529846, 0.5420894622802734, -0.12920810282230377, -1.1227973699569702, 1.6569187641143799, -0.2001953274011612, -2.8205225467681885, 0.022803369909524918, 0.338579922914505, 2.5052173137664795, 0.6098515391349792, 0.8664754629135132, -0.19071123003959656, 0.21248312294483185, -0.5441117882728577, -0.0998476967215538, 0.3931627571582794, -0.8354147672653198, -1.5450959205627441, -0.6475629806518555, -0.09711208939552307, -0.5447452068328857, 0.3249765932559967, -0.23962156474590302, -0.745415449142456, 1.0501588582992554, -1.4206370115280151, 0.7643924355506897, -0.7836238741874695, -1.197097659111023, -2.3267083168029785, -1.6704816818237305, -0.23812702298164368, 0.03475337475538254, 0.9523739814758301, -1.0763908624649048, -1.0272061824798584, 0.4632088840007782, 1.0087276697158813, -1.1410692930221558, -2.154142379760742, 0.04558292031288147, -2.397571325302124, -0.5246090292930603, 0.2840456962585449, -0.007386508397758007, 0.4711531698703766, 1.6607296466827393, 1.428731918334961, -0.8606869578361511, 0.9099767208099365, 0.49535730481147766, 1.0672709941864014, 0.22753314673900604, -0.8254271745681763, 2.013746500015259, 0.03703077882528305, -0.44420233368873596, -0.03477225452661514, 0.6597126722335815, -0.40766629576683044, 0.2468346208333969, 2.319474935531616, 1.0882558822631836, 0.06721016019582748, 1.2349072694778442, -0.8177740573883057, 0.8365272879600525, 2.934006690979004, -0.061964135617017746, -0.15141460299491882, -1.765152931213379, -0.029440589249134064, 1.5029946565628052, -1.2176612615585327, 0.8325538635253906, 0.6582402586936951, -1.7436577081680298, -1.3358248472213745, -0.10539206862449646, 0.9357097744941711, -0.7005248665809631, -0.2829878628253937, -0.4876621663570404, -0.4549585282802582, 0.6301930546760559, -0.3171252906322479, -1.0153573751449585, 1.8013513088226318, 0.3786712884902954, -0.02789492718875408, -0.04439777508378029, 0.11272728443145752, 1.0904327630996704, 0.5473248362541199, -1.1177116632461548, -1.053755760192871, -1.2354267835617065, -0.2810388505458832, -2.021061420440674, 0.042380087077617645, 0.8056175708770752, -1.8331592082977295, 0.2634986639022827, 1.111403226852417, -0.7390339374542236, -0.30379658937454224, 0.9421320557594299, 1.8772796392440796, 0.7220240831375122, 0.07179027795791626, 1.009834885597229, 1.2016681432724, -0.6817150115966797, 0.04887191578745842, -1.3972269296646118, -0.08881893754005432, 1.5440486669540405, -0.6537746787071228, 1.9090659618377686, 0.7124927043914795, -1.3110153675079346, -1.856056571006775], [-0.627964437007904, 1.0835548639297485, 0.384644091129303, 0.7552704215049744, -0.2998144328594208, -0.1214398443698883, -1.5870072841644287, 2.037091016769409, -0.4679473042488098, -0.24498265981674194, 0.11690642684698105, -1.7175136804580688, -0.9473596215248108, 1.5310863256454468, 0.8645756840705872, -0.3631545901298523, -0.5570490956306458, 0.19835232198238373, -1.1682850122451782, -0.4031485915184021, 0.3820918798446655, 1.3975861072540283, -0.17734117805957794, 1.0341423749923706, -0.9974586963653564, -0.017080653458833694, 0.35921990871429443, -1.1909819841384888, -1.0967988967895508, 1.3186534643173218, -0.4242081344127655, -1.422353982925415, 0.2949509620666504, -1.2177104949951172, 1.301274299621582, 0.153676837682724, 0.004360675346106291, -1.505680799484253, -0.8126237392425537, -0.687018632888794, -1.3393534421920776, -0.05617726594209671, 0.453544020652771, -1.536078691482544, -0.33567842841148376, 0.9611737728118896, 0.5866326689720154, -1.0202707052230835, -0.6621379256248474, -0.7239784598350525, -1.5163542032241821, -1.914667010307312, -2.138082981109619, -0.6916690468788147, 0.48721081018447876, 0.7063286304473877, 0.6552985310554504, 0.9504641890525818, -1.061964988708496, -0.83064204454422, -0.3677349090576172, 0.3521345257759094, -1.6806962490081787, 0.5229538679122925, -0.583512008190155, -0.750231921672821, -0.4278965890407562, -0.6824392676353455, -0.74693763256073, -0.07908838242292404, -1.0115622282028198, 0.6561566591262817, 0.9248068928718567, 1.4174222946166992, -0.19840553402900696, 0.21421308815479279, 1.1696844100952148, 0.8721579313278198, -1.0758271217346191, -0.9319563508033752, 1.4195038080215454, 1.238996148109436, -1.298869013786316, 0.5812202095985413, 0.9119312763214111, -1.2690714597702026, 0.6734047532081604, -1.1989374160766602, 0.39077675342559814, 0.7744876146316528, -1.73423171043396, 2.149338722229004, -0.42078346014022827, -0.7858088612556458, 0.43066710233688354, -0.04360068216919899, -0.917728841304779, 0.03593747317790985, -1.3588930368423462, 0.135457381606102, 0.43445464968681335, -0.03647228330373764, -1.078447937965393, 0.20742768049240112, 0.7324117422103882, 0.9263865947723389, -0.4574117362499237, -0.05508998781442642, 2.5246517658233643, 1.603281021118164, -0.34370145201683044, -0.036299146711826324, 1.800060510635376, 1.4110232591629028, -1.540123701095581, 0.9766262173652649, 0.1350398063659668, 1.384905457496643, -0.01589183881878853, -0.4223010241985321, 0.40368932485580444, -0.29432767629623413, 0.6384117603302002, -0.08754687756299973, 0.7683387994766235, 0.5541046261787415, -1.462242841720581, 1.8821641206741333, 2.312270402908325, 2.054276466369629, 0.15279385447502136, -1.0389506816864014, -1.4745445251464844, 1.4436753988265991, 0.3197081983089447, -1.2017343044281006, -1.0575504302978516, 0.8747465014457703, -3.914482593536377, 0.10913057625293732, 0.17014341056346893, 0.65582275390625, 2.759103775024414, -1.103380799293518, 0.28497353196144104, 0.44284477829933167, -1.3702367544174194, 0.14838559925556183, 2.5689995288848877, 0.031422123312950134, -1.8935155868530273, 0.1254539042711258, 1.8156694173812866, 0.38040006160736084, 0.44221803545951843, 0.07056249678134918, 0.9489323496818542, 1.4845447540283203, 1.5409560203552246, 0.2385871410369873, 0.21525472402572632, 0.36005884408950806, 1.0129859447479248, -1.91653573513031, 1.466359257698059, 0.4729571044445038, -0.014793564565479755, 0.08603323996067047, 0.4501883387565613, 0.6418763995170593, -0.3320070803165436, -1.172067642211914, -0.7795078754425049, 0.2800014913082123, 1.6146936416625977, -0.5811068415641785, -0.3043576776981354, -0.8246806859970093, 1.0036747455596924, -1.1309335231781006, 1.487345814704895, -1.518682599067688, 0.7768102288246155, -0.48136621713638306, 1.138602614402771, -1.0566468238830566, -2.3497748374938965, -1.4273526668548584, 0.14261068403720856, 0.46305304765701294, -1.2463682889938354, 1.379949927330017, -1.187044382095337, -0.7826859951019287, -0.34304189682006836, 0.06995120644569397, -0.8895712494850159, -1.946984887123108, 0.7130375504493713, -0.008093693293631077, 0.2949525713920593, 1.0819417238235474, 0.18685798346996307, -1.2221134901046753, 0.7466714382171631, -1.3277106285095215, 0.6344346404075623, -0.1961013376712799, 0.4190521538257599, -1.1428197622299194, -0.5417752861976624, -0.18791215121746063, 1.9065864086151123, 0.5268811583518982, -0.9104669094085693, -0.6314418911933899, 0.022481312975287437, -0.3635648787021637, 1.056089162826538, -0.6228007674217224, 0.04603113606572151, 0.4564287066459656, -0.9766725301742554, -0.04992516711354256, -0.03392724692821503, 2.3932125568389893, -0.04148722440004349, 1.0286402702331543, 0.2030823975801468, -0.29372504353523254, 0.04776226356625557, 2.846893310546875, -1.6357249021530151, -1.3757401704788208, -0.01193056907504797, 0.6549350023269653, 0.8396132588386536, 1.9595919847488403, -0.10941359400749207, -0.7955843210220337, 0.6215274930000305, -0.7598397731781006, -1.3949484825134277, -0.6089565753936768, 1.128232479095459, 1.251946210861206, 1.2258427143096924, 0.32124337553977966, -1.0517878532409668, -0.14523504674434662, -1.1820155382156372, -0.7295940518379211, -0.21129854023456573, 1.2242450714111328, 1.2976011037826538, 0.35623598098754883, -0.4643256962299347, 0.29158711433410645, -0.8340629935264587, -0.6023221611976624, 1.0876610279083252, 1.4861948490142822, 0.20929597318172455, 0.689134955406189, 0.19521713256835938, -0.6969189643859863, -1.3195717334747314, 0.7973875403404236, -0.5222710371017456, -0.1211976557970047, 0.7370515465736389, -1.6277951002120972, -0.4882800877094269, -0.2903410792350769, 0.10144586861133575, -0.6641829013824463, 0.8364695310592651, -0.47761261463165283, 0.5165152549743652, 0.39591851830482483, 0.5482805371284485, -0.1407797783613205, 0.14557833969593048, 2.473632335662842, -0.32572829723358154, -0.49333930015563965, -0.09631980955600739, 0.11699873954057693, 0.15561898052692413, -0.11033844947814941, 0.427034467458725, 1.4218448400497437, 0.0598972849547863, -0.7009016871452332, -0.533954381942749, 0.21601811051368713, -1.0955901145935059, -0.5366354584693909, -0.056876663118600845, 0.29557275772094727, -1.9176207780838013, 0.34121349453926086, -1.9658305644989014, 0.2558647692203522, -0.5626657009124756, 0.7459803223609924, 0.31768089532852173, 0.5572078824043274, 0.7500272393226624, 0.40427863597869873, 0.6785163879394531, -0.30543795228004456, 2.1093835830688477, -1.2348321676254272, -0.9128445982933044, -0.564080536365509, -1.128100037574768, -1.5383415222167969, -0.7070574164390564, 0.5988638401031494, 0.443430095911026, 2.1340959072113037, -0.45097678899765015, 0.761338472366333, -0.7150754928588867, 0.833924412727356, 0.3566759526729584, 1.3758372068405151, 1.0300469398498535, 0.12264430522918701, -0.5138396620750427, 0.5654797554016113, 0.5748860836029053, 0.579945981502533, -0.3456145226955414, -0.768580973148346, -1.379656195640564, -1.645263433456421, 1.2496072053909302, 1.6116669178009033, 0.6902499794960022, -0.47264018654823303, -0.3869117200374603, 0.8215745687484741, 0.2168605625629425, 3.109718084335327, 0.5806560516357422, 0.24587558209896088, -0.9600009322166443, 0.2826508581638336, -0.31739866733551025, 0.9921872615814209, -1.7620353698730469, -0.4186657965183258, -1.5174912214279175, -1.478115200996399, 0.27551305294036865, -0.659730851650238, -2.0112290382385254, 1.0541592836380005, -0.3105788826942444, 0.1656576544046402, -0.3687900900840759, 1.1203486919403076, 0.5098943710327148, 1.2435158491134644, -1.0557169914245605, -0.11076861619949341, -0.42240774631500244, -1.089869737625122, 2.338404417037964, 0.9078041315078735, -0.343086838722229, 0.4088977575302124, -0.5584971904754639, -0.1268106997013092, -1.367147445678711, 0.4529171884059906, 1.1471837759017944, -0.5810088515281677, -0.37327656149864197, -1.1897727251052856, -2.550272226333618, -1.2733012437820435, 0.7568856477737427, -0.48569536209106445, -0.38821467757225037, 0.20041266083717346, 0.0790143758058548, 0.07860149443149567, -1.56663978099823, -1.3604671955108643, -0.36042147874832153, -1.6249037981033325, 0.5188567638397217, -0.1404893398284912, -0.03385143354535103, -1.2879807949066162, -0.5139071941375732, -0.3831843137741089, 0.05513232201337814, 0.5360486507415771, 0.07846663892269135, -0.36613985896110535, -0.5160488486289978, -0.16114118695259094, -0.7351657748222351, -0.37549224495887756, 0.6237872838973999, -0.4964412450790405, -0.5980221629142761, 1.234618902206421, -1.1647236347198486, 0.060969915241003036, 0.06805869191884995, 1.575400948524475, -1.0265878438949585, -0.40934211015701294, -1.3594998121261597, 1.2468037605285645, -2.00789213180542, 0.20236137509346008, 0.05220016837120056, 1.789109230041504, 0.3683546483516693, -0.7623940706253052, -0.36315473914146423, 1.3357067108154297, 0.7762508392333984, 0.6573960185050964, -0.9467690587043762, -0.2913951575756073, -0.3233215808868408, -0.7043718695640564, -0.1066024899482727, 1.745722770690918, 1.0752729177474976, 0.3439827263355255, -1.1769263744354248, -0.34316471219062805, -1.292435646057129, 1.2625306844711304, 0.8984125256538391, 1.4176461696624756, -0.8088840246200562, -0.5737990140914917, 1.134677767753601, -0.6408995389938354, -0.6144700050354004, -1.3800610303878784, 1.1123487949371338, -0.3483104407787323, 1.3403637409210205, -0.07345494627952576, -0.3570784032344818, 2.0078494548797607, 1.9221487045288086, -0.45657315850257874, 0.21461527049541473, -0.8750192523002625, 0.44929811358451843, 0.6766321063041687, 0.09089241921901703, 1.5918086767196655, -1.2831671237945557, -1.110080599784851, 0.5857871770858765, 0.5862568616867065, -0.05008444935083389, 0.1588868647813797, 0.2958719730377197, -2.4646995067596436, -0.13565146923065186, 0.9810178875923157, -0.5580490231513977, -2.100811004638672, 1.3405951261520386, 0.7060393691062927, 0.06049107760190964, -2.026442766189575, 0.35130107402801514, 0.14052000641822815, -0.3385406732559204, 0.01613514870405197, 2.912473201751709, -1.7320904731750488, -1.0888034105300903, 1.3201549053192139, -0.17081010341644287, 1.2256896495819092, -1.142820954322815, 0.41471174359321594, 1.348669171333313, 0.6345400810241699, 0.5026305913925171, -0.9764532446861267, 0.18661057949066162, -0.44569990038871765, -0.20598827302455902, -1.651438593864441, 1.835724115371704, -0.4950293004512787, 2.0076167583465576, 0.9446296691894531, -0.9266113042831421, -0.9288564920425415, 0.6211977601051331, -0.6164077520370483, -0.9834733009338379, -1.2009859085083008, -0.7085171341896057, -0.010490004904568195, -0.6212589740753174, -0.21976329386234283, 0.30128201842308044, 0.48262131214141846, 0.741793692111969, -0.055795252323150635, 0.5097863674163818, -0.3863973021507263, -0.09183748066425323, 0.9493343830108643, -0.3574671745300293, -1.988510251045227, 0.7869112491607666, -0.7776943445205688, -0.028528347611427307, 0.7375665307044983, -1.3230055570602417, -2.106367349624634, -0.10144386440515518, -2.0984292030334473, 0.5990612506866455, 0.06870230287313461, 0.29245245456695557, 0.5404133200645447, -0.5122746229171753, 0.2023988664150238, 0.5750672817230225, -0.38929301500320435, 2.1490421295166016, 0.2174249291419983, 0.8111264109611511, -0.1063186302781105, 1.035886287689209, -0.6422082185745239, -0.28494784235954285, -1.0388602018356323, 0.6087712049484253, -0.9167538285255432, 2.7594242095947266, -0.8586658239364624, -1.5815279483795166, 0.9699845910072327, 0.6811772584915161, 0.14830783009529114, 0.6688656806945801, -1.0542770624160767, -0.31766176223754883, -0.22626495361328125, 0.9407504796981812, 0.4416548013687134, -0.2671452760696411, -0.13361231982707977, -1.011927604675293, -0.6136090755462646, 0.3246888518333435, -1.075215220451355, 0.08167941123247147, 0.9272996783256531, 0.29740390181541443, 1.731541395187378, 0.5660063028335571, 0.24101802706718445, -0.49810904264450073, -0.39946937561035156, -0.6392253637313843, 1.8056752681732178, -2.5363752841949463, 0.7690320014953613, 0.3604290783405304, 1.5083000659942627, -1.1401848793029785, -0.6542601585388184, 0.6729437112808228, -0.9111950397491455, 0.07031507790088654, -0.04582475125789642, -0.753075122833252, 0.5315230488777161, -2.380016326904297, 2.2024271488189697, 0.011857863515615463, -1.3310843706130981, -0.4824661910533905, -1.244739055633545, 0.17416273057460785, 0.6097529530525208, 0.5022430419921875, 1.2800064086914062, -0.9877648949623108, 0.9965912103652954, 1.2148375511169434, -1.5934721231460571, 0.8576595783233643, -0.10420510917901993, 0.5417808890342712, -0.3422800302505493, 1.6607121229171753, -1.0048221349716187, 0.29810699820518494, -0.1782987415790558, -0.513260006904602, -0.6338313221931458, 0.7922642827033997, -0.9514159560203552, 0.5801305174827576, 1.3674367666244507, 0.4930177628993988, -1.0571922063827515, 0.1307268589735031, -1.4266479015350342, -0.1824202984571457, -0.41655614972114563, 1.7379448413848877, -0.1402519941329956, 0.949391782283783, 1.588300347328186, -0.32268887758255005, 0.5045629739761353, 0.23324044048786163, -0.5601702332496643, -0.2822285294532776, 0.5303056240081787, 0.3804469406604767, 0.5978689193725586, 0.8752124309539795, 1.7919834852218628, -1.205386996269226, 0.25464946031570435, -0.023624412715435028, 0.1412307173013687, -1.1435366868972778, 0.6083356142044067, 0.27357280254364014, 0.9438263773918152, 0.9720872640609741, 2.533322811126709, -0.7850703001022339, -1.659663438796997, -0.33724603056907654, -0.8331708312034607, 0.21786606311798096, -0.044857896864414215, 0.48525506258010864, 0.7152917385101318, -0.9817719459533691, -0.6394028663635254, -0.07350229471921921, 0.07450316846370697, -0.4215497076511383, 1.3828061819076538, -1.077111840248108, 1.0084158182144165, 1.0181127786636353, 0.3146781027317047, 1.001224398612976, -2.2373340129852295, -0.753857433795929, 0.005888479296118021, -0.18695984780788422, 1.2128485441207886, -0.009084772318601608, 1.2203774452209473, -0.5777378678321838, -1.1096965074539185, 0.43447282910346985, -0.5280783772468567, 0.004185358062386513, 0.7278375625610352, 1.2826008796691895, 0.8028438687324524, -0.15208542346954346, 2.0362389087677, -0.7159824967384338, -0.5672074556350708, -0.62626713514328, 1.156793236732483, -0.900417149066925, -0.09423678368330002, -0.15837177634239197, 0.6958527565002441, 1.240394949913025, 0.5023539662361145, 0.021360980346798897, -0.20138314366340637, -1.3417913913726807, -0.20395419001579285, 0.07222220301628113, -0.15970414876937866, -1.092714786529541, 0.7593312859535217, 1.1551827192306519, 0.2882867455482483, 0.5749387145042419, 0.47205543518066406, -0.44123518466949463, 0.95506751537323, -0.41082528233528137, -0.6573694944381714, 0.7012050151824951, -0.6001594066619873, 0.7451053857803345, -1.2505587339401245, -0.993187665939331, -0.5968751907348633, 1.1408710479736328, -0.39882639050483704, 1.676181674003601, 1.7017067670822144, -0.008553244173526764, 0.5757290124893188, 0.7122408151626587, 0.7713316082954407, -1.2842425107955933, -0.3593931496143341, 0.36742570996284485, -1.097723126411438, -1.773635983467102, -0.7291784882545471, 0.66086745262146, 1.343186855316162, 0.15959155559539795, 0.34366950392723083, 0.16779825091362, 1.927364468574524, 1.1656858921051025, -1.536604404449463, 1.7667609453201294, 1.4670963287353516, -0.7074753642082214, -1.3207148313522339, 0.16171149909496307, 0.4096643924713135, -1.1426535844802856, -0.9662691354751587, -1.3636362552642822, -0.13931620121002197, -0.9320986866950989, -0.5549868941307068, -0.9232315421104431, -0.6878657341003418, 1.0027055740356445, -0.82142174243927, -1.1023882627487183, 0.8621575236320496, 0.48073065280914307, 0.36594337224960327, -0.20733526349067688, -0.19183126091957092, -0.3110942840576172, -0.2760395109653473, 1.2270110845565796, 0.0173201784491539, 0.1843760907649994, 0.32386571168899536, 0.972018837928772, 0.1247134655714035, -0.6569780111312866, -1.440279245376587, -0.7280519604682922, -0.26384082436561584, -0.1604904681444168, 0.6698013544082642, 0.48944342136383057, -0.35922491550445557, -1.5591181516647339, 0.5233166813850403, -1.0494649410247803, 0.2174614518880844, -0.5187954902648926, -0.4169779419898987, -1.160871148109436, 0.550808310508728, -0.7405946254730225, -1.273630142211914, -0.41930124163627625, 0.7729421257972717, -0.9342714548110962, -1.9044227600097656, 0.4896022379398346, -1.184292197227478, -1.3324270248413086, 0.4239213168621063, 1.7871763706207275, 2.898451089859009, 1.151060938835144, -0.8102116584777832, 0.009353932924568653, -1.456594467163086, -1.2336369752883911, -0.40133988857269287, -0.5017595291137695, 1.515987753868103, 1.209107756614685, 2.071476936340332, 1.8070108890533447, -0.17825569212436676, -0.0634780153632164, -1.6997506618499756, 0.3409702479839325, -0.8433704972267151, -0.4481576979160309, -0.39945220947265625, -0.3611900210380554, 0.29761749505996704, 1.6600430011749268, -1.216427206993103, -1.1582975387573242, 0.46211355924606323, 0.07479001581668854, -0.6618840098381042, -0.09851142764091492, -1.5695569515228271, -1.0861393213272095, -0.18159864842891693, 1.5811872482299805, 1.1868842840194702, -1.6232556104660034, 0.6582837104797363, -0.9820818305015564, -1.6906379461288452, 1.369113564491272, 1.7242364883422852, -0.43693017959594727, 2.1418747901916504, -1.1163303852081299, 0.6392499208450317, -0.44493526220321655, -2.331855058670044, -2.546449661254883, 0.2169039398431778, -0.5346017479896545, -0.9010955691337585, -0.8957200646400452, -0.560000479221344, -0.0797128677368164, -0.38078051805496216, 0.6468953490257263, 1.7606240510940552, 1.2830380201339722, -0.07686533033847809, 0.32990890741348267, 0.19791238009929657, 1.7645323276519775, 2.2838032245635986, 1.598137378692627, -0.96830153465271, 0.458690881729126, -0.006742413621395826, -1.5248419046401978, -0.6994541883468628, 0.23845753073692322, -0.7234207987785339, 1.4349265098571777, 0.40776991844177246, -0.9487479329109192, -0.3751538395881653, -0.6813111901283264, 0.46435803174972534, -1.239691972732544, 0.16339661180973053, 0.4846174418926239, 2.1225178241729736, 2.421027898788452, 1.2588366270065308, -0.027962898835539818, 0.785286009311676, -0.44959571957588196, 0.7634978890419006, 1.3643518686294556, -0.6377160549163818, -0.9486215114593506, -1.585866928100586, -0.141890287399292, -1.8364931344985962, 1.5619903802871704, 0.7617915272712708, -1.0319099426269531, -0.6553217172622681, 0.07072123885154724, 0.9370303153991699, -0.7311863899230957, 1.1204582452774048, 1.5435588359832764, -0.7330933809280396, 1.3166747093200684, 0.5794505476951599, 0.8577637076377869, 1.9697834253311157, 1.7027168273925781, -0.37187010049819946, -1.703641414642334, 1.631943941116333, -0.63321453332901, 0.5415272116661072, 0.3913908004760742, -0.5689058899879456, 0.3339078724384308, 0.4742320775985718, 1.1532052755355835, -0.7705332040786743, -0.9967809915542603, 3.651756525039673, -0.039251454174518585, -1.1046488285064697, -0.15446053445339203, -1.8374722003936768, 0.7499880194664001, -1.1946080923080444, 1.1891741752624512, -0.045667752623558044, -0.744377613067627, -0.7736422419548035, 1.0510553121566772, -0.49309372901916504, 1.7107878923416138, -0.646979033946991, -0.5437629818916321, -0.03058338165283203, -1.500065565109253, 1.7025201320648193, -0.1284077912569046, -0.6349002718925476, -0.8363847732543945, 0.9882004261016846, -1.1004836559295654, -0.7104130983352661, 0.03590364009141922, -0.5618426203727722, -0.17967186868190765, -1.0851233005523682, 1.6007860898971558, -1.1514034271240234, -0.46648332476615906, -0.8085355162620544, -0.7737689018249512, -1.507651686668396, -0.349911093711853, -0.6627440452575684, -1.7654551267623901, 1.3688987493515015, 1.0609419345855713, -0.3837096691131592, -1.6401927471160889, -1.3249599933624268, -0.6278937458992004, 1.315600872039795, 0.6264278888702393, 0.004250769969075918, 0.0984836146235466, -0.06812065094709396, -0.3323838710784912, 0.9627737402915955, 0.8126522302627563, -0.6058405637741089, -0.14770066738128662, 0.6932283639907837, -0.21188488602638245, -0.54573655128479, 2.179236650466919, 0.7300480008125305, 1.1089423894882202, -0.6619897484779358, -0.5577254891395569, -1.1328911781311035, 1.1056134700775146, -0.9187014698982239, -1.1375977993011475, -0.7583540678024292, -1.4681626558303833, 0.7007511258125305, 1.4484308958053589, -0.16875946521759033, -0.458713561296463, -0.49035370349884033, -0.303211510181427, 0.5120990872383118, 0.4298904836177826, -0.6434716582298279, 3.2376718521118164, 1.4241019487380981, -0.15697960555553436, 0.9061394333839417, 1.9720836877822876, -2.9498307704925537, 1.21010160446167, 0.41509854793548584, -0.2388964146375656, 0.1832016408443451, 0.6506820321083069, -0.18433992564678192, 0.158778116106987, -0.8002858757972717, -0.6541411280632019, -1.537567138671875, -1.250905156135559, -0.13843482732772827, 0.1713499128818512, -1.2850522994995117, 0.9854658246040344, 0.4810355007648468, -0.8339357972145081, -1.6712745428085327, 0.5575880408287048, -1.571712613105774, 0.909125566482544, -0.38086363673210144, -0.04520741477608681, -0.6303396224975586, 0.25435203313827515, 0.10950589179992676, -0.2928330898284912, 1.11493980884552], [-0.016101770102977753, -1.1032496690750122, -0.5706996321678162, -0.4234776496887207, -1.5403813123703003, -1.836363434791565, -0.14153000712394714, -0.20536865293979645, 0.49736690521240234, -0.3112318515777588, -0.22260692715644836, 0.20541585981845856, 0.3955319821834564, -0.45593199133872986, -0.29311925172805786, 0.48781266808509827, -0.2150222808122635, -0.22609847784042358, -0.02497304417192936, -0.5489519238471985, 0.8968691825866699, -1.0925047397613525, -1.4542121887207031, 0.5058824419975281, 0.02177509292960167, -0.8616557717323303, 0.9995445609092712, -1.2598209381103516, -0.9907940626144409, 1.5644811391830444, 0.08743040263652802, -2.789163112640381, -0.1109284907579422, -1.347078561782837, 1.3539948463439941, -0.6501187682151794, -0.536337673664093, -0.5475088357925415, -1.723468542098999, -0.5533998608589172, 2.101999044418335, 1.0795975923538208, -0.1439245045185089, 0.8769883513450623, 0.38635018467903137, 2.0717382431030273, -1.1216188669204712, 0.041496653109788895, 1.406386137008667, 0.33058348298072815, -0.1758110672235489, -1.5501610040664673, -0.8154261708259583, 0.1126788854598999, -0.9573181867599487, -2.233875036239624, -1.321968913078308, -0.6864007711410522, 1.5968310832977295, 1.1497670412063599, 0.9732248783111572, 0.9740445017814636, -0.7131245136260986, 1.6649667024612427, 0.8491682410240173, -2.429945707321167, 0.9457047581672668, -0.8633593320846558, -1.0670990943908691, -0.7261242866516113, -1.5623377561569214, 0.5533828139305115, 0.6362190246582031, -1.0031161308288574, 1.4899026155471802, 0.10456392914056778, 0.4489107131958008, 0.14254845678806305, 0.4373202323913574, -1.5401270389556885, 0.2927345633506775, -0.08029124140739441, -1.1548715829849243, -1.3520314693450928, -0.05959279090166092, -0.5744588971138, -0.11980657279491425, -1.4478570222854614, -0.6087491512298584, 0.007696951739490032, 1.4739634990692139, 1.514647126197815, -0.28515753149986267, -1.8832679986953735, 0.423328161239624, 0.07318484038114548, 1.8781318664550781, 0.8958815336227417, 1.500178337097168, -1.1909211874008179, 0.2503063678741455, -1.035316824913025, 0.5808984041213989, 0.8927692770957947, -2.132446050643921, 0.8599579930305481, -0.39660242199897766, 2.5021448135375977, -1.2695379257202148, -1.6899574995040894, 1.5692909955978394, -1.404984712600708, -1.4953006505966187, -0.47604143619537354, -0.29252225160598755, 0.3484102487564087, -2.0096867084503174, -0.11942100524902344, 1.0809160470962524, -0.9869633316993713, 1.1233365535736084, -1.0469894409179688, -0.9504230618476868, -0.9172991514205933, -0.3902049660682678, 0.011216588318347931, 0.13970395922660828, -0.9779959917068481, 0.3011227548122406, -1.1589754819869995, 0.45665472745895386, -2.0630786418914795, 0.9480264782905579, 0.0328054241836071, 0.03669184073805809, 0.35017886757850647, 0.023707613348960876, 0.9188428521156311, -1.021653413772583, -0.4897976219654083, 0.2704961597919464, 0.2170521318912506, 1.6527363061904907, 2.8998935222625732, -0.8113426566123962, 1.5720901489257812, -1.9345459938049316, -0.3842568099498749, -0.28345414996147156, 0.38297605514526367, -1.1006873846054077, -0.25695666670799255, 0.18333205580711365, -0.550322949886322, 0.06194122135639191, -0.4892350137233734, -0.250922828912735, 0.6900684237480164, -0.9678189754486084, 0.5943750143051147, 0.18435901403427124, 2.301074266433716, -0.7517837882041931, -0.16127397119998932, 0.0017628147033974528, 1.2924072742462158, -0.7552857398986816, 0.1830449104309082, 0.6742204427719116, 1.0762709379196167, 1.6634914875030518, 0.5829326510429382, 0.16605931520462036, 0.5166242718696594, -0.2478935867547989, -0.27986398339271545, 0.14283490180969238, -1.1084545850753784, -0.09674738347530365, 0.43973469734191895, 0.7570317387580872, 1.2614154815673828, -0.14526772499084473, 1.1229569911956787, 1.1943467855453491, 1.2906033992767334, 0.5359403491020203, 0.7059557437896729, 0.8454205989837646, 1.5215288400650024, -1.3201992511749268, 1.7974802255630493, -0.02901506796479225, -0.07855687290430069, 1.7558045387268066, 0.47707757353782654, 0.5740247368812561, -0.9032556414604187, -1.4782142639160156, 0.41527292132377625, -0.7292929291725159, -1.6265009641647339, 0.995274543762207, -1.1363964080810547, -0.5859695672988892, -1.5858343839645386, -0.40524160861968994, 0.04740868881344795, 0.3835662007331848, -0.014968175441026688, -0.7508120536804199, -0.4822250008583069, -1.2465349435806274, -0.779211163520813, -0.9531223177909851, 0.8252608180046082, 1.4868589639663696, -1.2579807043075562, 0.31522417068481445, 0.4197484850883484, 0.938582181930542, -0.0681094080209732, -1.7629557847976685, -0.09563273191452026, 0.26842445135116577, -0.5913283228874207, -0.01995391957461834, -0.7483486533164978, 1.612718105316162, -0.4735491871833801, 0.2866556644439697, -0.6964971423149109, -0.15230849385261536, 0.15637274086475372, -2.0603086948394775, 0.6762574911117554, -0.2524777352809906, 0.8710241913795471, -0.6944255232810974, -0.3878839313983917, -0.10852976888418198, -1.439821481704712, 0.8449853658676147, -0.24864859879016876, -0.1886947602033615, -1.072689175605774, -0.6284573674201965, 1.3506524562835693, -0.6882303357124329, -0.5790923237800598, 0.6065437197685242, -0.4715522527694702, 0.4736177623271942, -0.775291919708252, -0.9606404900550842, 0.11004779487848282, 0.4113089144229889, -0.7115499377250671, 0.8353567123413086, 0.46787065267562866, 1.0633646249771118, 0.4240599274635315, 1.8345775604248047, -1.9116209745407104, 0.03227992355823517, 0.688449501991272, 0.1277330368757248, 0.6027853488922119, -0.09048425406217575, 0.3822404146194458, 0.12448704987764359, 0.4911949932575226, -0.6494352221488953, -0.9386875629425049, 1.0926353931427002, -0.45348668098449707, -0.1200576052069664, 0.35576584935188293, -0.9783333539962769, 0.6060189008712769, -1.0024789571762085, 0.037631191313266754, -1.3788856267929077, -2.6695291996002197, -0.9201458692550659, 1.0918134450912476, -1.1405543088912964, 1.3177155256271362, -0.6582916975021362, 1.0786833763122559, -1.8739769458770752, 0.07806272804737091, 0.6758734583854675, -0.8946640491485596, 0.12668916583061218, -1.0358028411865234, -0.3476620614528656, -0.16049957275390625, -0.417693555355072, 1.6686103343963623, 0.3839688301086426, -1.5633820295333862, -1.3970422744750977, -2.7316365242004395, 0.2474452704191208, 1.4019407033920288, -0.588297426700592, 1.4397567510604858, -0.8876505494117737, -2.578845500946045, 0.46032968163490295, -0.7374218106269836, 0.6784864068031311, -0.08775362372398376, 0.5416068434715271, 0.2861554026603699, 0.950207531452179, -2.1735219955444336, 0.7567906975746155, -0.7519411444664001, -0.014655347913503647, 1.751127004623413, 1.5012304782867432, -0.5610925555229187, 0.19696055352687836, 2.6460485458374023, 0.47872400283813477, -0.1828811913728714, -0.3023284673690796, -0.09732279926538467, -0.9430682063102722, 0.240905299782753, -0.7885441184043884, 1.0366199016571045, 0.8119317889213562, 0.40000438690185547, 0.44516587257385254, 0.4275891184806824, -1.3962048292160034, 0.7293752431869507, 0.5599597096443176, 1.6501243114471436, -0.1952313631772995, -0.9487943053245544, 0.6028519868850708, -1.4468916654586792, -0.5508254766464233, 0.12841464579105377, 2.19162654876709, 0.24853907525539398, -1.259904146194458, -1.1261857748031616, -0.021731318905949593, 0.027329448610544205, -0.3948124647140503, -1.2799938917160034, -0.36900389194488525, -0.5575584173202515, 1.1422734260559082, 0.625756561756134, -0.02386203035712242, 0.7107748985290527, 1.0352517366409302, 1.3081387281417847, 0.7330204844474792, -1.1562860012054443, -0.18981803953647614, 1.3578381538391113, -0.6308886408805847, 0.9587095975875854, -0.3851195275783539, -1.6309760808944702, 1.1776599884033203, 3.1878390312194824, -0.917565643787384, 0.9682602286338806, -0.6194796562194824, -0.3758104145526886, 0.7928724884986877, 1.084092617034912, 0.359976589679718, 0.9905733466148376, -0.46909719705581665, 0.5181828141212463, 0.0898509994149208, -0.6100331544876099, -0.8189154267311096, 1.1886166334152222, -2.2229480743408203, -1.1164857149124146, -0.6346691250801086, 0.1377744972705841, -0.2024918794631958, 0.7427828311920166, -1.5383280515670776, 0.07964502274990082, -0.7659894824028015, 0.5830792188644409, 1.2684040069580078, -0.6825307011604309, -0.13091078400611877, -0.11597029119729996, -0.8599066138267517, -0.9642435312271118, -0.9294813275337219, 1.374549388885498, -0.9381450414657593, 0.541331946849823, -0.02991533651947975, -0.28860247135162354, 1.0588854551315308, 0.6583285927772522, -0.35656192898750305, -0.47213584184646606, 0.15923866629600525, 0.44928744435310364, -0.6588212847709656, 0.13285261392593384, -0.29972508549690247, -0.16882866621017456, -0.7965894937515259, -1.6998728513717651, -0.6949072480201721, 0.39442309737205505, -2.5323574542999268, 0.26751086115837097, 1.6523786783218384, 0.307304322719574, 0.049252111464738846, 0.6154882311820984, -1.482284426689148, 0.6975841522216797, 0.7151647806167603, 0.6233437657356262, 0.7923105359077454, -1.551621437072754, -0.6074849963188171, -0.33820146322250366, -1.6415263414382935, 0.44488975405693054, 0.318477064371109, -0.4735862612724304, 0.2892102003097534, -1.8017596006393433, 0.49940621852874756, 0.11048179864883423, -0.2760733664035797, 0.6678177118301392, -0.24425309896469116, 1.100799322128296, 1.2757008075714111, -1.9495036602020264, 0.186876118183136, -0.08681540936231613, -0.5087077617645264, 1.3992220163345337, 0.6012535691261292, 0.6809560656547546, 0.2767132520675659, -1.8839973211288452, 0.6385915875434875, -2.7043421268463135, -1.6558464765548706, 1.0914021730422974, -0.9869503974914551, -0.24200348556041718, -0.2922801971435547, -0.7011156678199768, 1.6211587190628052, -0.7110239863395691, -0.6354140639305115, -2.1769895553588867, -0.1300433874130249, 0.7510601878166199, 0.8838821649551392, 1.075779676437378, 1.6600598096847534, -0.8498347401618958, 0.1423695683479309, -0.181539848446846, 0.28430235385894775, -0.9597336053848267, 0.8310263752937317, -0.7100737690925598, 0.6260221004486084, -0.030980214476585388, 0.6333259344100952, 1.0502598285675049, 0.1411103457212448, -0.10559612512588501, -2.7284467220306396, -0.35587185621261597, 2.215496063232422, 1.395015835762024, -2.096351146697998, 1.0383901596069336, -0.6478768587112427, 0.0907391607761383, 0.18392378091812134, 1.2522474527359009, 1.9692236185073853, -0.22746771574020386, -1.087271809577942, -0.273340106010437, 0.16297605633735657, -0.24828624725341797, 1.2254719734191895, 0.5585426688194275, -0.6149774789810181, -0.8788090348243713, 0.770320475101471, -1.3170170783996582, -1.9487136602401733, -1.4074394702911377, 0.8956608772277832, 0.49617427587509155, -0.07619703561067581, -0.8114539384841919, -0.4306194484233856, -0.868702232837677, 0.4787711501121521, -0.47896310687065125, 1.565726399421692, -1.1443490982055664, 0.5033355355262756, 1.1731104850769043, -0.9449333548545837, -0.20093323290348053, 0.8524343967437744, -0.596546471118927, -0.7263458371162415, -0.669421374797821, -0.3927878737449646, -0.055846117436885834, -1.4958065748214722, -1.165223479270935, 1.292864441871643, -1.1088676452636719, 2.0272223949432373, -1.291125774383545, 0.3680400848388672, 0.2509033679962158, -0.9728649258613586, -1.4694710969924927, 0.01379865501075983, -0.8179543018341064, -0.32034486532211304, -0.18670158088207245, -0.7906848192214966, 1.4975786209106445, 0.39551788568496704, 0.16423285007476807, 0.8527042865753174, -1.3030359745025635, -1.433566927909851, 0.9870691299438477, 0.5738270878791809, -2.0627734661102295, 1.3751531839370728, 1.024897575378418, -1.7818835973739624, -0.22573159635066986, -0.6131439208984375, -0.0674547478556633, 0.17756249010562897, -0.24103616178035736, -0.4552173912525177, 1.3808043003082275, 1.1602650880813599, 0.9173872470855713, -1.0087531805038452, -0.3713199496269226, 0.8455030918121338, -2.3289198875427246, -0.41841402649879456, 0.9492019414901733, 0.3485758304595947, -0.12305480986833572, -2.406526565551758, 0.468097984790802, -1.7157557010650635, 0.8250693082809448, -0.2519169747829437, -0.11909664422273636, 0.49740365147590637, -0.5487379431724548, 0.6602335572242737, -0.4994034767150879, 1.1081401109695435, -0.941858172416687, 1.1127315759658813, 1.5845608711242676, 0.3999136686325073, 0.05840763449668884, -0.3129439353942871, 0.6829718947410583, 1.3687578439712524, -0.8155568242073059, 0.6261290311813354, -1.6053178310394287, -2.371757745742798, -0.601105272769928, 0.089458167552948, -0.5484052896499634, 0.6397237777709961, -1.1157230138778687, -0.6894057393074036, -1.2381315231323242, -1.8918830156326294, 0.48229527473449707, -1.370102882385254, -0.05596305802464485, 0.3354036509990692, -1.4467686414718628, -0.6655352115631104, -0.01335945911705494, 0.6864240765571594, -2.1399314403533936, -0.41456010937690735, -0.2637069821357727, 0.042169239372015, -1.5321784019470215, -0.06231136992573738, -0.9168458580970764, -1.6470023393630981, 0.44158825278282166, -0.08867232501506805, -1.00360906124115, 0.4324744641780853, -2.4957659244537354, 2.5134544372558594, 0.10358253121376038, -0.6092285513877869, 0.7762085795402527, -0.27615490555763245, -0.10762172937393188, 0.053605061024427414, 0.8834680914878845, -2.054328203201294, 0.694561779499054, -0.02525678649544716, 0.6403585076332092, 1.8114417791366577, 1.2582212686538696, 0.5163790583610535, 0.35114312171936035, 0.8088217377662659, -0.45200732350349426, 0.19577738642692566, -0.7612494826316833, 0.9624749422073364, -1.2212002277374268, -0.3405914604663849, -0.6084038615226746, 1.0689696073532104, -0.0794757753610611, 0.3653966188430786, -0.9855419397354126, -0.8115100860595703, 0.4072384536266327, -0.7102333307266235, -0.7429718375205994, -0.4304646849632263, 1.1200462579727173, 0.4631355404853821, 0.1171094998717308, 0.5171381235122681, 1.128624677658081, -1.8682433366775513, -0.3632529675960541, -0.03616473078727722, -0.18272145092487335, -0.7851797342300415, -0.5047743916511536, -0.7751030325889587, 1.3720265626907349, 0.04103473946452141, -0.2021871656179428, 0.1329014003276825, 0.015592514537274837, -1.442487120628357, 1.0533580780029297, -0.14802393317222595, -0.9792439937591553, -1.6114940643310547, 1.1329935789108276, 2.3126180171966553, 0.5898080468177795, -0.8526005148887634, 0.6816146969795227, -1.3627854585647583, 1.522566795349121, 0.12217049300670624, 0.17953018844127655, 1.4035422801971436, -1.1923555135726929, 0.5286887288093567, 1.0185375213623047, 0.16805967688560486, -1.4143052101135254, -1.2630631923675537, 1.2517330646514893, 0.43185749650001526, -0.20395727455615997, -0.17142947018146515, -0.7906214594841003, -2.4813833236694336, 0.18940016627311707, 1.9469127655029297, 0.8015130162239075, -0.4107722043991089, 1.2518376111984253, 0.4832788109779358, -0.301322340965271, 0.14107653498649597, -0.37944740056991577, 0.77226722240448, 0.5406875014305115, -0.1724659651517868, 0.22102545201778412, 0.9290643930435181, 1.57175874710083, 0.6750653386116028, 2.3387370109558105, -1.07270085811615, 0.6563220620155334, -1.0227906703948975, -1.0645359754562378, -0.12515437602996826, 1.241536259651184, -1.2242242097854614, 0.8769581317901611, 1.9055237770080566, -0.29376721382141113, -1.500778317451477, -0.4095248878002167, 0.9592909216880798, 1.6133959293365479, 1.0034455060958862, 0.6322826743125916, -0.2636549472808838, -0.2406858652830124, -1.448326826095581, -2.1370251178741455, 1.915476679801941, -0.06038501858711243, -0.7001457214355469, 0.17177046835422516, 1.1106727123260498, -1.5120247602462769, -0.49970167875289917, -0.24064189195632935, -0.8558449745178223, -1.0909631252288818, -0.9231095314025879, 1.8156095743179321, 0.8984196782112122, -0.47629669308662415, 0.7215116024017334, 0.7421002984046936, 2.442122459411621, 0.24065424501895905, -0.6457731127738953, -0.14000500738620758, 0.7494999170303345, -0.18622323870658875, -0.18731307983398438, 1.5104100704193115, 0.463342547416687, 0.6491069197654724, 1.1993342638015747, -0.7542007565498352, 1.0945484638214111, -1.1572160720825195, -0.26318296790122986, -0.7919862866401672, 1.6152067184448242, -0.8168976902961731, -0.6888705492019653, 0.05269523337483406, -1.60688054561615, -0.08560514450073242, -0.9711456298828125, 0.8453009128570557, -0.6994450688362122, 0.5587139129638672, -1.4112812280654907, -0.07248390465974808, 0.5272434949874878, -0.6110139489173889, 0.7573443651199341, -0.6976343989372253, -1.2189490795135498, -0.641377329826355, 1.353935956954956, -0.2749471962451935, 0.7034404873847961, -1.4286460876464844, -1.5961112976074219, -0.17434494197368622, 1.251747965812683, 0.19981828331947327, 1.2316877841949463, -0.05300597473978996, -0.7257066965103149, 0.47441890835762024, -0.9653071761131287, 0.6465574502944946, 1.3442631959915161, 1.3141437768936157, 0.30742934346199036, 0.8516988158226013, -0.21415752172470093, 1.5433413982391357, 0.11483091861009598, -1.0577318668365479, 0.8662533164024353, -1.8018181324005127, -0.16073191165924072, -0.5313653945922852, 1.3580999374389648, -0.8737199902534485, -0.8893668055534363, 0.7183794379234314, 0.6402719616889954, 0.32786843180656433, 0.6329842209815979, -0.5021569132804871, -0.6207706332206726, -1.521694540977478, -0.460280179977417, 0.9571176767349243, -0.08527279645204544, -0.7495058178901672, 1.5915132761001587, -0.012708538211882114, -0.48827534914016724, 0.07934240996837616, -0.7906970977783203, -1.6285051107406616, 0.6354938745498657, 0.43337395787239075, 0.09708969295024872, -1.1710925102233887, 0.5664390921592712, 1.7417922019958496, 0.26769304275512695, 0.3451695144176483, 1.3582770824432373, -0.36295172572135925, -2.1337437629699707, 1.4190753698349, -0.6945119500160217, 0.5095689296722412, 1.0590355396270752, 0.5846542716026306, 1.7160323858261108, 0.07457489520311356, -1.359569787979126, 1.3270599842071533, 0.719406008720398, -0.5915451049804688, -1.4204484224319458, -0.44592440128326416, -0.16300782561302185, 0.8349626064300537, -0.8643306493759155, 1.7527261972427368, -0.3320700228214264, -1.42898428440094, 2.1095011234283447, 0.19594529271125793, 1.3186513185501099, 0.2106820046901703, -1.5170084238052368, 0.10964465886354446, 0.0816236138343811, 1.4483898878097534, -0.8518964052200317, 0.7119476795196533, 0.3065226376056671, -0.6865458488464355, -0.34958383440971375, 0.9380534887313843, -3.1070706844329834, 0.20475497841835022, -0.4080235958099365, 0.7338942289352417, 0.3933088779449463, 0.8639931678771973, 0.00687818368896842, -0.09669335186481476, -0.8408724069595337, 0.8884884119033813, 0.5047568678855896, 1.5846278667449951, -1.3493320941925049, -0.8504447340965271, -0.8721824288368225, 0.16807273030281067, 0.39986222982406616, -0.10776083916425705, -0.5870695114135742, -0.3239210247993469, 0.6325304508209229, -1.4001904726028442, 0.5647727847099304, -1.4488104581832886, -0.30891650915145874, 0.4796072244644165, 1.464638590812683, -0.6890432834625244, 0.7748773097991943, 1.1268179416656494, 0.6925533413887024, -2.2641584873199463, -0.725126326084137, 1.098684310913086, 0.9801503419876099, 0.05494188889861107, -1.3016949892044067, -1.2819340229034424, -0.4419955909252167, 1.6910980939865112, 0.8583577871322632, -0.43128255009651184, 0.020865490660071373, -0.3638487160205841, -0.28065407276153564, -1.1224695444107056, -0.15333645045757294, 0.8878589272499084, -0.8646811246871948, -0.6401971578598022, -0.956993043422699, -1.2140926122665405, -2.0109951496124268, 1.3718401193618774, -0.48190024495124817, -0.04902345687150955, -0.07559745758771896, 1.0112038850784302, -0.5782373547554016, -1.0213156938552856, -1.4132845401763916, 0.9211273193359375, 0.4703702926635742, 0.02485482767224312, -0.4220356047153473, 0.10751592367887497, -0.3560149371623993, -2.2315359115600586, -0.561231791973114, -0.10957861691713333, 0.2863244414329529, 0.20225653052330017, 1.404176115989685, 0.13275602459907532, -0.38472193479537964, -2.2602548599243164, 1.2413512468338013, -0.6503466367721558, -0.046739742159843445, 0.18068315088748932, 1.3607842922210693, 0.19227324426174164, 1.0536013841629028, -1.6027485132217407, -0.6240076422691345, -0.3634757399559021, -0.6227537393569946, 0.06503786891698837, 1.1016651391983032, -1.5912907123565674, 0.46895697712898254, 0.19494253396987915, 0.06860761344432831, -0.5359432697296143, 0.9171288013458252, 1.3143750429153442, -0.4000455439090729, 0.6410078406333923, -0.12182623893022537, 0.24066132307052612, 0.9270933866500854, -1.1659531593322754, -0.6376261711120605, 1.6288585662841797, 0.6184627413749695, -0.4311470687389374, 1.1971334218978882, 0.041456565260887146, -0.9009518027305603, -0.6693879961967468, -0.4539439082145691, -0.5075640678405762, 0.17437109351158142, -1.4186183214187622, -0.5533304214477539, -2.0015177726745605, 2.489285469055176, -1.2631703615188599, 0.6677475571632385, 0.812127411365509, 0.31344112753868103, -1.4709031581878662, -0.6173833608627319, -0.03385768085718155, -0.11192179471254349, 0.6450088620185852, -1.6196069717407227, 2.526560068130493, -0.5175787210464478, -0.3388805389404297, -1.1026250123977661, 0.3060436546802521, 1.7006852626800537, 0.33156606554985046, -0.5488519072532654, -0.15135596692562103, -0.83355313539505, 1.2254053354263306, -0.6769537925720215, 0.8412107229232788], [-0.49292513728141785, 0.11064367741346359, 0.5546032786369324, 1.3204420804977417, -0.48440465331077576, 0.0708218663930893, 0.018774988129734993, -0.7438104748725891, 0.7373496890068054, -0.5806649923324585, 1.5989426374435425, -0.4462302029132843, -0.7706959843635559, -0.22746336460113525, 0.8802053928375244, -1.9160335063934326, -0.07747901976108551, 0.30681875348091125, 0.07773366570472717, -2.761540412902832, -1.2502634525299072, 0.6335811018943787, 0.713914155960083, 1.4187296628952026, 0.8264935612678528, 0.15602777898311615, -0.24530921876430511, 0.5731768608093262, -1.6169534921646118, -0.15909171104431152, 0.5560173392295837, -0.3329026997089386, 0.5994437336921692, 0.5315418243408203, 0.14570148289203644, 0.4781273901462555, -1.0979176759719849, 1.0715841054916382, -0.5175142288208008, 0.24548999965190887, 0.38747408986091614, 0.02821575105190277, 1.4919629096984863, -1.4632971286773682, -0.728576123714447, -0.8164042234420776, 0.6871656179428101, 1.2538148164749146, 0.6337616443634033, -1.861879825592041, -0.9259824156761169, -1.1184221506118774, -1.02601957321167, -0.7695522904396057, -2.8767576217651367, 1.0633012056350708, 1.4999865293502808, 1.3924704790115356, -0.9948480725288391, -0.2430555373430252, 0.08318299800157547, 1.5935821533203125, 0.11352398246526718, -0.39660924673080444, -0.13036935031414032, -0.4681134819984436, -1.5113087892532349, 1.997031331062317, -1.461269736289978, 0.8318474888801575, -1.3824046850204468, -1.8048566579818726, -1.0402008295059204, 1.0120638608932495, -0.767921507358551, -1.78937828540802, -1.4893847703933716, 0.34455886483192444, 2.2337710857391357, 3.0436716079711914, -1.6191195249557495, -0.9519217014312744, 0.13338083028793335, -0.1245805025100708, 0.8539630770683289, -0.038615331053733826, -1.021384835243225, 1.8641669750213623, -0.6562432646751404, 0.3502984344959259, -0.09689589589834213, 0.3875150978565216, -0.4437318742275238, -0.2949722111225128, 0.1685400754213333, -0.0005441243411041796, -0.5155572295188904, 0.601327657699585, 1.4930943250656128, -2.1390280723571777, 0.01066548004746437, -0.23603151738643646, -0.8911901712417603, -2.328944444656372, -0.12122047692537308, 0.20100799202919006, 1.7605713605880737, 1.061401605606079, 0.30638840794563293, 1.2645633220672607, -0.4345593750476837, 1.702284336090088, -1.3757100105285645, 0.5333431363105774, -0.9258889555931091, -1.1532034873962402, -0.0398310087621212, 0.29288968443870544, 0.6455294489860535, -0.4226837456226349, 1.2052557468414307, -0.25080761313438416, -0.7399823069572449, 0.1486268788576126, 0.863723874092102, 1.941835641860962, 1.802319049835205, -1.2859729528427124, 0.0634821355342865, -0.5237443447113037, -0.5613089203834534, -0.4781026840209961, 0.6889777183532715, 0.025407977402210236, -0.4957160949707031, -0.7264125347137451, 0.4487285315990448, -0.8134219646453857, 0.4548012614250183, -1.252892255783081, -0.4170539975166321, -0.9237720370292664, -0.9851115345954895, 1.4451881647109985, -1.265743374824524, 0.5154699683189392, 0.47610563039779663, -0.16694921255111694, 0.24942144751548767, 0.614995539188385, 0.41578277945518494, -1.7112619876861572, 0.5443799495697021, -0.07559167593717575, 0.8863516449928284, 0.4028645157814026, -0.10009902715682983, -1.0407496690750122, -0.7330716252326965, -0.12895217537879944, 0.4197905659675598, 1.8246334791183472, -0.2927663326263428, -0.8888775110244751, -0.36992037296295166, 0.996727466583252, -0.09683936834335327, -0.8569872379302979, -0.2691502869129181, -0.6488509774208069, 1.0179532766342163, -0.3781201243400574, -0.8107778429985046, 1.4486240148544312, -0.8330348134040833, 1.4464263916015625, -1.2419646978378296, -0.5082354545593262, -0.3831743001937866, 0.08983129262924194, 0.15401174128055573, 1.4731427431106567, 0.6842696070671082, -0.7796298861503601, -0.9540655016899109, -0.42188331484794617, -0.5328525304794312, -0.21275173127651215, 1.1374351978302002, -0.6035168766975403, 0.32043102383613586, -0.3745819330215454, 1.7738953828811646, 0.6839049458503723, 0.9389052391052246, 0.41716268658638, -0.14192146062850952, 1.9306524991989136, 0.03820756822824478, -0.07060495764017105, -0.7717458605766296, -1.0324634313583374, 1.1179986000061035, -0.10363294929265976, -0.44646477699279785, -1.3403269052505493, -0.9349601864814758, 0.7222351431846619, 0.7990732192993164, 0.6499410271644592, 0.7507789731025696, 1.0971599817276, 0.11553905159235, 0.9346996545791626, -0.6463735699653625, -0.028731605038046837, -0.7357459664344788, -0.202993243932724, 0.8233324289321899, -0.8109844326972961, 0.873028576374054, -0.04420538991689682, -0.8457027077674866, 1.663732647895813, 1.5590908527374268, 0.26232779026031494, -0.25645583868026733, -0.26208236813545227, -1.0811960697174072, -1.3124172687530518, -0.7790048122406006, 1.0447540283203125, -0.6626830697059631, -0.26486319303512573, 2.3809845447540283, 0.09642191231250763, 0.5341491103172302, -0.5300674438476562, 0.05003289878368378, 0.3263261914253235, -0.2436131238937378, -0.4231855869293213, 0.21284674108028412, -0.9962484240531921, -0.792700469493866, 0.32801491022109985, 0.5100671648979187, -0.7411481738090515, 1.4348337650299072, -0.2657841444015503, 1.0839687585830688, -0.9261820316314697, 0.9160295724868774, -0.7517991065979004, 0.4754713773727417, -0.35529759526252747, -0.8832924365997314, -0.2980266809463501, -0.5306843519210815, 1.961349606513977, 1.2709853649139404, 1.0233417749404907, 0.7931607365608215, 1.4133226871490479, -0.3293409049510956, -1.245090365409851, 0.6668155193328857, -0.17788384854793549, 1.486293911933899, 0.8062130212783813, 0.6327754259109497, 0.41965243220329285, 1.1527880430221558, -1.589052438735962, -0.6705604791641235, 0.07087642699480057, -1.0732629299163818, -0.4014798402786255, 0.08820156753063202, 0.6788935661315918, 1.2026313543319702, 1.8039450645446777, 1.1582213640213013, -2.1488187313079834, -1.829541802406311, 0.6218430995941162, 1.3073885440826416, 0.3061259388923645, -0.3939083218574524, -0.09060902893543243, -0.16574744880199432, -1.4468834400177002, -0.80332350730896, 1.5840166807174683, -0.3278374969959259, -0.4626312553882599, 0.5927080512046814, -0.4957907199859619, -1.4222761392593384, 1.7215344905853271, 0.04983683302998543, -0.26258981227874756, -0.6163421273231506, -0.94549560546875, -0.36286622285842896, 0.6391426920890808, 1.2265087366104126, -0.5888801217079163, -0.8816034197807312, 0.09330500662326813, 1.3879673480987549, -1.0005115270614624, 0.9365670680999756, -0.6368058323860168, -0.23648935556411743, 0.8629626631736755, -2.1779913902282715, -0.10091117024421692, 1.9637500047683716, 0.6880168318748474, -0.7920352816581726, -0.7943728566169739, 1.24810791015625, 0.09404110908508301, -1.0158461332321167, -1.0142329931259155, -0.11923786252737045, 1.8222750425338745, -1.6379061937332153, 0.2012612521648407, -0.18102259933948517, 0.004228061530739069, 0.5309371948242188, 0.9733183979988098, -1.3225942850112915, -0.42328038811683655, 0.6047156453132629, 0.24188019335269928, 1.2432923316955566, 1.7963206768035889, 0.4490186274051666, 0.45630210638046265, 0.12271568924188614, 0.519160270690918, -0.621100664138794, 1.1116060018539429, 0.5865622758865356, -0.7536278963088989, 0.7611339688301086, 1.778952956199646, 0.03690560534596443, 0.42637285590171814, 0.140427365899086, -1.0316162109375, -0.5166050791740417, -0.6857572793960571, -0.2970939874649048, -0.6931946873664856, -0.34492263197898865, -0.01907777599990368, 0.6339794397354126, 0.14359815418720245, -1.5998320579528809, 0.274612694978714, 1.1958425045013428, -0.1667860746383667, 0.8629952073097229, 0.5799541473388672, 0.7938274145126343, 1.1180955171585083, 0.3888765573501587, -0.20064863562583923, -0.28445789217948914, 0.6266955137252808, 0.972227156162262, -0.2547481060028076, -0.41964462399482727, -0.11645564436912537, 1.035753607749939, -1.444655179977417, -1.7490034103393555, 0.10696223378181458, 0.9021325707435608, 0.8805592060089111, -1.3066874742507935, 1.3354498147964478, 0.09123065322637558, -0.1586293876171112, 0.14340819418430328, -1.1606630086898804, -0.2792075574398041, -0.6017987728118896, -1.0813013315200806, 1.3766815662384033, 0.9738235473632812, 1.6281015872955322, -0.2817179560661316, 0.023728827014565468, -0.18172292411327362, -0.12014140188694, 0.1384810209274292, 0.33554404973983765, 0.0630408450961113, -0.15119530260562897, -1.137363314628601, -1.0734533071517944, -1.6018105745315552, 1.0967881679534912, -0.8719004392623901, 0.8743532299995422, -1.9208712577819824, 1.4224116802215576, 0.8805791735649109, -0.19259022176265717, -1.944765329360962, -0.835237979888916, -0.3746214509010315, 2.1046812534332275, 0.2736316919326782, -1.0949245691299438, -1.4969955682754517, 1.6032825708389282, -1.0008325576782227, -0.5136346817016602, 1.4126462936401367, 0.42464035749435425, 0.5540817379951477, -0.26935845613479614, 1.5638388395309448, -0.910037636756897, 0.5241620540618896, 0.937321126461029, 0.15790535509586334, 0.18962182104587555, -0.8652241826057434, -1.5775688886642456, 1.5636017322540283, 0.43345606327056885, -1.5486972332000732, -0.06105382740497589, 0.9223330616950989, -0.9746488332748413, -0.9579366445541382, 3.0779247283935547, -0.24268703162670135, 1.4477200508117676, 1.0142929553985596, -0.8199057579040527, 0.6267086863517761, 0.2703360617160797, -3.085414171218872, -0.2657089829444885, 0.1570197194814682, -0.8329066634178162, -1.9292566776275635, 1.5851563215255737, 0.332361102104187, -1.1274042129516602, -0.18453505635261536, 0.10404877364635468, 1.9866321086883545, 0.604061484336853, -1.5573539733886719, 0.05390775203704834, 1.0906343460083008, 0.3341844975948334, 0.6634832620620728, -0.9928861260414124, -1.9187514781951904, 0.6060947775840759, -0.8695365786552429, 0.483734667301178, 1.0728541612625122, 0.06928976625204086, -1.6386839151382446, -1.8913432359695435, -1.0377262830734253, -1.8624223470687866, -0.9414803385734558, 1.302378535270691, 1.7834211587905884, -1.0031166076660156, 0.19323492050170898, -2.416156530380249, -1.3750109672546387, -0.4541110396385193, 1.6118333339691162, -1.030206561088562, 0.7824757099151611, -0.8817042708396912, 0.7674856781959534, 1.1567076444625854, 2.1738057136535645, -1.240291714668274, 0.7470627427101135, 0.5314226746559143, -0.5721633434295654, 0.4147099256515503, -0.5697625279426575, 0.4448053240776062, -0.5106474757194519, -0.8174092173576355, 2.793416976928711, 1.2618284225463867, 0.5813935995101929, 0.6940984725952148, 1.800750494003296, -0.3368052840232849, -0.6945024728775024, -0.2978745400905609, 0.3868755102157593, 1.6765624284744263, 0.16734816133975983, -0.4520772099494934, -1.3035376071929932, 0.620776355266571, -0.12679727375507355, -0.5106279253959656, 1.4340957403182983, -0.748013436794281, 0.43043118715286255, -0.4466719329357147, -0.20977815985679626, 1.6738108396530151, 1.185544490814209, -2.451568603515625, -0.9746825695037842, -1.1705378293991089, -0.10845598578453064, 0.6701310873031616, -0.22257225215435028, -0.599201500415802, 0.8228272199630737, -1.1963810920715332, 0.32180359959602356, 0.452730655670166, -1.433748722076416, -0.08353681862354279, 0.6077786087989807, 0.08514463156461716, 0.5643494129180908, 1.5268770456314087, -1.0750377178192139, -0.2781499922275543, -0.048664942383766174, -0.5407892465591431, -0.8959143757820129, -0.4136684834957123, 1.4033474922180176, 0.8580883145332336, 0.5638409852981567, -1.0238986015319824, 0.3393527567386627, 0.21054168045520782, 0.18956544995307922, -1.3284982442855835, 0.17428776621818542, -0.31016772985458374, 0.16443879902362823, -0.45785316824913025, -0.6366748809814453, 2.742485523223877, -0.3470635712146759, -0.543827474117279, 0.6433145999908447, -2.3708009719848633, -1.1491754055023193, 0.17755144834518433, -1.7343558073043823, -0.40920257568359375, -0.6459535956382751, -0.17330624163150787, 1.1710224151611328, -0.7400804758071899, 0.1538344770669937, -0.6927738189697266, 0.6288066506385803, 0.11663311719894409, -0.23493756353855133, -0.413604199886322, 1.4105470180511475, -2.3574562072753906, 0.5502652525901794, -0.9046117067337036, -0.9347638487815857, -0.5878477096557617, -1.1382266283035278, -0.3540704548358917, -1.7416499853134155, 0.5667730569839478, -0.5466415882110596, -0.0888676866889, -1.100635051727295, 0.8635885715484619, -0.7610403299331665, -2.0154781341552734, -0.4045792520046234, -1.324276089668274, -1.111770510673523, 1.6525944471359253, 0.7857857942581177, 0.01570659689605236, -0.5577915906906128, -1.0054659843444824, 0.9202521443367004, -0.307110458612442, -0.33957117795944214, -0.3446648418903351, -1.286752700805664, -1.185558557510376, 1.8452765941619873, -0.7922164797782898, -0.0028434928972274065, -1.239330768585205, -0.6915305256843567, 0.4966842234134674, 0.24531587958335876, -2.968510389328003, 1.5029499530792236, 0.03677473962306976, 1.3000624179840088, 0.4296128451824188, 0.5582166314125061, 0.2428632527589798, -0.3607695400714874, 2.4850850105285645, 0.32597586512565613, 1.6637508869171143, -0.16391688585281372, -0.21628037095069885, 0.5810880064964294, 1.5596165657043457, -0.7917810678482056, 1.9691096544265747, 1.223151683807373, 0.8581725358963013, -1.3227450847625732, -1.3466323614120483, -0.12815621495246887, 0.9446828365325928, -0.548255443572998, -0.6866933107376099, 0.8654043078422546, -0.41652730107307434, -0.11830705404281616, -1.9525938034057617, -0.3932384252548218, -0.3146892488002777, 0.7212458252906799, 0.34556496143341064, 2.0577468872070312, -0.19656100869178772, 0.7965452671051025, 0.0018197911558672786, 1.0659631490707397, -0.6356983184814453, -0.49463948607444763, 0.8181506991386414, 0.9367024898529053, -1.9584609270095825, -0.7705548405647278, 0.07036716490983963, 1.112544059753418, -0.6977141499519348, 0.485493928194046, 0.9069474339485168, 0.4547763466835022, 1.021336317062378, -0.4770839810371399, 0.2280336320400238, -1.4799699783325195, -0.8578293919563293, 2.0916430950164795, -0.4447864592075348, -1.048100471496582, 1.894513487815857, -0.4422213137149811, 0.6066393852233887, -2.4491865634918213, 1.157948613166809, -0.8202368021011353, 0.6434131264686584, 0.6163312196731567, -0.1361507773399353, -1.458964228630066, -0.41835662722587585, -0.1859840601682663, 1.2049140930175781, 0.6587486863136292, -0.6286011338233948, -0.30808374285697937, 0.7029210925102234, 0.43088391423225403, -1.0426745414733887, -0.16584716737270355, 1.9906870126724243, 1.6123334169387817, -0.32125648856163025, 1.0921075344085693, -1.1271090507507324, -0.3892548382282257, -1.1833661794662476, 1.5031291246414185, -0.980760395526886, 1.1671861410140991, 0.5667567849159241, -0.36783257126808167, -1.3648263216018677, 1.5543274879455566, -1.4833227396011353, 0.6845420598983765, -1.4105890989303589, -1.566762089729309, 0.1347390115261078, -1.9498053789138794, -0.7386294007301331, 1.1178791522979736, 0.018539417535066605, 0.7027307152748108, 0.8943949341773987, 1.5612668991088867, 0.8291605710983276, -0.3601430356502533, 0.7923973202705383, 2.5537712574005127, 0.7952120304107666, -0.031762395054101944, 0.024700963869690895, -1.3245699405670166, 0.4018898904323578, -0.8471181988716125, -0.2222309559583664, -0.6275496482849121, -0.370381623506546, -0.742712140083313, -0.1738343983888626, 1.1287848949432373, 0.3984607756137848, 0.3715928792953491, -0.10632769018411636, -0.1334100067615509, 1.0485689640045166, 0.022728292271494865, 0.19810684025287628, 0.48239320516586304, -1.7238024473190308, -0.6863049268722534, 0.7513926029205322, 0.042452551424503326, -0.24738779664039612, 1.0668100118637085, 1.119810938835144, 0.15196789801120758, -1.3168652057647705, 0.40468716621398926, -1.0293316841125488, -0.4727422595024109, 0.2386608123779297, -0.7261819839477539, 0.396198034286499, -0.6880984306335449, 1.8578174114227295, 0.7879039645195007, -1.59726881980896, -0.008186222054064274, 0.8840769529342651, 0.41174444556236267, -0.7179776430130005, 0.8441110849380493, 1.0528782606124878, -0.49822917580604553, 0.48736438155174255, 0.02896064892411232, -0.6655575037002563, 0.024061281234025955, 0.3460826277732849, -0.7010321617126465, 1.707826018333435, -0.3538029193878174, 0.7437981963157654, 0.6030830144882202, 0.6481333374977112, -0.22079434990882874, -1.2951265573501587, -0.3241405487060547, -1.7749567031860352, -1.408490777015686, 0.6297495365142822, -0.5416991710662842, 0.0028159457724541426, -0.8001775145530701, -1.3979240655899048, -0.9139333963394165, 1.0827804803848267, -0.3809957206249237, -0.026675593107938766, -0.4409106969833374, -0.17384536564350128, -0.24259179830551147, 0.22477692365646362, 0.456291139125824, 0.41126179695129395, -0.5361025333404541, 0.4435422420501709, -0.9649974703788757, -1.2162249088287354, -0.05211568623781204, -1.9541419744491577, -0.6350882649421692, 0.4138133227825165, -0.5239057540893555, -0.8901355862617493, 0.034525852650403976, 0.597274661064148, 0.2572545111179352, -1.101143479347229, 0.3621492087841034, 2.47218656539917, -0.017182456329464912, -0.6201872825622559, 0.8719080686569214, -1.6013528108596802, -1.2904627323150635, -0.609472393989563, -0.326781690120697, 0.1919853389263153, 0.5363581776618958, 0.3663368225097656, -1.1183220148086548, 0.1679234504699707, 0.27363547682762146, -1.1767282485961914, -1.908862590789795, -0.5394113063812256, -1.5011136531829834, 0.8635486960411072, -0.979494571685791, -1.0558531284332275, -0.6316026449203491, -0.15702958405017853, -1.1512953042984009, -0.35899820923805237, 0.6443993449211121, 0.08184319734573364, 0.45906734466552734, -0.30863264203071594, -1.379644751548767, 1.0448895692825317, -0.19921867549419403, 0.6725959181785583, 0.5581859946250916, 0.643953800201416, -1.2290666103363037, -1.0109339952468872, -0.608301043510437, -1.9240158796310425, 0.08272017538547516, 0.8050378561019897, -0.614963710308075, 1.147961974143982, 0.6698590517044067, -0.9803078770637512, 0.26294440031051636, -1.1456327438354492, 0.339516818523407, 1.3237448930740356, -0.016638239845633507, 0.1710631251335144, -0.028587929904460907, -2.14579176902771, -0.005078200250864029, 0.8763531446456909, 0.14131063222885132, 0.4245243966579437, -0.1397218257188797, 0.9448854327201843, -1.1942713260650635, 1.0577797889709473, -2.9856109619140625, 0.38412871956825256, 1.1003063917160034, -1.512839674949646, -0.7316933870315552, 0.3279648423194885, 0.8152135014533997, -0.5065902471542358, -0.7273796796798706, 0.2601783275604248, -0.29894423484802246, -0.48294687271118164, -1.5332305431365967, 0.55030357837677, -0.0207743588835001, 1.0757718086242676, 0.21653662621974945, -0.2603958249092102, 1.1366612911224365, 0.805590808391571, 0.31084686517715454, 2.1829874515533447, 0.23168642818927765, 0.9274979829788208, -0.43998241424560547, 0.9531651139259338, -0.6608346700668335, -0.823599100112915, -0.8288410305976868, 0.6463015079498291, 0.6959832906723022, -0.463969886302948, -0.9252161979675293, 0.36751753091812134, 0.7317088842391968, -0.9746776223182678, 0.5297664999961853, -0.20851337909698486, 1.603200078010559, 1.174846887588501, 0.6215554475784302, 0.314270555973053, 1.2380033731460571, 0.32653525471687317, 0.5461830496788025, 0.9004220962524414, 0.6335061192512512, -0.05199151858687401, -0.8086526989936829, -0.07425263524055481, -1.0360239744186401, 0.4501706063747406, 0.2673857808113098, 0.07713960111141205, -0.8488085269927979, 0.3629545569419861, 0.1605430692434311, 2.442065954208374, -1.5579243898391724, 2.8427846431732178, -0.038094013929367065, 1.1218289136886597, 0.8272983431816101, 0.07643074542284012, -0.8950297832489014, -1.3717186450958252, -1.3092615604400635, 1.5965932607650757, -0.5274437665939331, -0.4255521893501282, 0.12888188660144806, -0.5165575742721558, -0.6991183757781982, -0.7806341648101807, 0.8835187554359436, -0.6034203171730042, -0.31494081020355225, -0.376499742269516, 0.1917591542005539, 1.2961230278015137, 0.41099438071250916, -0.3100648522377014, 0.6708762645721436, -2.084278106689453, 0.004796845838427544, -0.12252978980541229, 2.5985796451568604, -1.3091143369674683, -0.9393002390861511, 0.09815634042024612, -2.390434503555298, -2.434232711791992, 1.7113386392593384, -2.580864429473877, 0.5112653374671936, 2.0420584678649902, -0.15758821368217468, -0.6583447456359863, 0.43284332752227783, -1.662911057472229, -0.7910123467445374, 0.8114708662033081, -1.1046812534332275, -0.858672022819519, 0.09663185477256775, -0.8923561573028564, 1.3843997716903687, -0.33365488052368164, 0.33789709210395813, 0.2818889319896698, 0.9018875360488892, 2.058772087097168, 0.6577832698822021, 0.22546221315860748, -0.6016316413879395, 1.336832046508789, 1.3146235942840576, -0.4482277035713196, 0.6116527915000916, -1.4378243684768677, -1.8775596618652344, 0.15605656802654266, -0.4903005361557007, -0.16374163329601288, 1.509920358657837, 0.6306675672531128, 2.7532899379730225, -0.4669541120529175, -1.1395241022109985, 0.6068929433822632, 1.3586814403533936, 0.9751875400543213, -0.9331966638565063, 1.9785947799682617, -0.9931090474128723, -2.1829190254211426, -0.48989543318748474, -0.6697072386741638, -1.0346930027008057, -0.9147364497184753, 0.08249012380838394, -0.5812541842460632, -1.1948165893554688, -1.9160792827606201, -0.3108486533164978, 0.8894281387329102], [0.22584280371665955, 0.5999598503112793, -0.029994675889611244, 0.04043568670749664, -0.8879063725471497, 0.6583958268165588, 1.1750776767730713, -0.7538344264030457, 0.12590532004833221, 1.4163029193878174, -1.0601364374160767, 0.9667457938194275, 0.30213215947151184, -0.8728389143943787, -0.3662242591381073, -0.6800675988197327, -0.7971158027648926, 0.40003177523612976, -0.3987143635749817, 0.10188616812229156, -0.44717517495155334, -1.128805160522461, -0.5900796055793762, 1.3148146867752075, 0.04615679755806923, -2.061640501022339, 0.23906204104423523, 1.0660032033920288, 0.1582491397857666, -0.9110809564590454, 1.5302449464797974, 1.1581472158432007, 1.6894155740737915, 1.2886251211166382, -0.18962550163269043, 1.0449141263961792, -0.08221731334924698, 0.4423648715019226, 0.21307793259620667, -0.0021626821253448725, 1.471901535987854, 1.5996251106262207, -0.773562490940094, 0.35111987590789795, 0.687197744846344, -1.1681890487670898, -0.006020369939506054, -0.25760841369628906, -0.8453834652900696, -1.156687617301941, -0.8400781750679016, -0.8876456022262573, 0.5215262174606323, 0.6085330843925476, 1.8178292512893677, -2.169182300567627, 0.04341653361916542, 1.2539242506027222, 0.7875270247459412, -0.023824241012334824, -0.039186347275972366, 1.4050689935684204, 0.24880360066890717, -1.3151137828826904, 0.408652126789093, 0.5406554341316223, -0.20326612889766693, -1.4891738891601562, -1.951674461364746, 0.9043705463409424, -1.2778170108795166, -1.165159821510315, 0.2267713099718094, 0.12403856217861176, -0.2537396550178528, 0.36202582716941833, 0.05704876035451889, -1.4573596715927124, -0.21211740374565125, -0.06194000691175461, 0.8519312143325806, -0.42736300826072693, -2.5537772178649902, -1.5959025621414185, 0.6503328680992126, 0.20372889935970306, 1.3219273090362549, -0.7617687582969666, -0.0285189226269722, 0.22169378399848938, 0.95784592628479, 0.14799299836158752, -0.8721058964729309, -0.45701879262924194, 1.153533697128296, 0.11204688251018524, 0.4512837827205658, -2.2210757732391357, -2.8690428733825684, -0.031947486102581024, 0.0776379406452179, -1.4229053258895874, 0.04151662066578865, 0.36251387000083923, -0.7702032327651978, -0.23471614718437195, -0.9835867881774902, 0.7814673781394958, -0.593531608581543, -0.42810919880867004, 1.8865188360214233, 0.5710838437080383, 1.2235283851623535, 0.6022597551345825, 0.3431762456893921, 0.2568330764770508, 1.4671622514724731, -0.27924439311027527, -0.6755567789077759, 1.0800909996032715, -0.8604314923286438, -0.9128068685531616, -0.6217520236968994, -0.059071656316518784, 0.49943697452545166, 0.42466092109680176, 0.9238821268081665, -0.8414080739021301, -0.39550644159317017, 0.9304952621459961, 1.1331533193588257, -0.15507081151008606, -0.3671242594718933, 1.0728110074996948, 1.6246329545974731, -1.4114989042282104, -1.605109453201294, 1.255645990371704, 0.3952329158782959, -0.46993449330329895, -1.1916106939315796, 1.393027424812317, -1.7249786853790283, -0.45954430103302, 1.5927865505218506, -0.2430637627840042, -0.10932587087154388, 1.615702748298645, -0.047119129449129105, -0.4009398818016052, 1.6430364847183228, 0.6878172755241394, -0.3169301152229309, 0.541781485080719, -1.0011900663375854, -0.08141648024320602, -1.0663195848464966, 0.2580946981906891, 0.708122968673706, -1.337191104888916, -1.9599491357803345, -0.4272863566875458, -0.9210618138313293, -0.11192535609006882, 0.15178132057189941, 0.9894904494285583, 2.5806124210357666, 1.5069580078125, -1.2275426387786865, 1.4176238775253296, -0.7527100443840027, 0.6950672268867493, 1.5340908765792847, 0.5926323533058167, -2.147691488265991, 0.47602343559265137, 0.4861496388912201, 1.262837290763855, -0.16773834824562073, 0.1333424597978592, 0.5920950770378113, 0.4365341365337372, -1.2304562330245972, -1.3419253826141357, -0.0013187238946557045, -1.2835270166397095, 1.1789157390594482, -1.0496479272842407, -0.7287265658378601, 0.2499818652868271, -1.466080904006958, -0.9274047017097473, -1.28690505027771, -0.2633827328681946, -1.2656570672988892, 0.8493031859397888, -1.9301360845565796, 0.48587965965270996, -1.223214030265808, -0.12453923374414444, -0.3866318166255951, 0.2339518964290619, -1.5699584484100342, 0.5490249991416931, -0.44086402654647827, -1.0301281213760376, 0.8629375100135803, -0.7453799247741699, 0.4984997808933258, -1.1223371028900146, -0.401978462934494, 0.7636911273002625, -2.617124557495117, 0.01813698746263981, 0.6034898161888123, 0.19357922673225403, 1.579594612121582, -1.0766184329986572, -0.3465636372566223, -0.030207674950361252, 1.1633580923080444, -1.1693333387374878, -0.17693686485290527, -0.664631187915802, 1.0364768505096436, -1.446835994720459, 0.4996676445007324, -0.68328458070755, -0.3671805262565613, -0.5547886490821838, 0.7978450059890747, 0.8331878185272217, -0.2288677841424942, -0.8801385164260864, 1.4913618564605713, 1.1019084453582764, -1.4381061792373657, 1.4900329113006592, 1.0954949855804443, 0.05541118606925011, -0.1631353348493576, -1.467040777206421, 0.07255050539970398, 0.8505041003227234, 0.924525797367096, 2.4562149047851562, 1.0510141849517822, -0.9902852177619934, -1.0160669088363647, -1.8431499004364014, 0.6983128190040588, -0.9738507866859436, -0.9341452717781067, -2.8717124462127686, -0.663226306438446, 1.0689080953598022, 0.967352032661438, -3.286883592605591, -0.9873602986335754, -0.10878219455480576, 0.6289989948272705, 0.7520120739936829, -1.1196482181549072, -0.4104328155517578, 0.48648467659950256, 1.5846459865570068, -0.7631190419197083, 1.3388944864273071, -1.6122405529022217, -1.771878957748413, 1.0975009202957153, -1.1579666137695312, -0.30988162755966187, -0.783827006816864, 1.2694833278656006, -0.03029690869152546, 0.620929479598999, -1.1234195232391357, -1.025571346282959, -0.33757975697517395, -0.17196528613567352, 0.17452576756477356, -0.3471473157405853, -1.0905358791351318, 0.7810971140861511, -1.0621212720870972, 0.48841241002082825, 0.0368056483566761, -2.0367302894592285, 0.6949676275253296, -1.2184051275253296, 1.0160927772521973, -0.6803038716316223, -3.5272421836853027, -0.2513636648654938, 0.914931058883667, 0.8818058967590332, 0.9724652767181396, 0.013715412467718124, 0.47429051995277405, 0.8057356476783752, 0.35794898867607117, 0.9116925001144409, 0.7788201570510864, 0.5063235759735107, 1.026917815208435, -0.3102046251296997, 1.4518184661865234, 1.5204930305480957, -0.04621351882815361, -1.4739900827407837, 1.7873789072036743, -0.5784057378768921, 1.4709887504577637, -1.9704416990280151, 0.9981690645217896, 0.3801678717136383, -0.3113904595375061, 1.0798124074935913, -1.0412635803222656, -1.2630589008331299, 0.524999737739563, 0.26165926456451416, -0.5568079948425293, -0.05781754478812218, 0.8141219019889832, 0.27073177695274353, -0.018733350560069084, 0.7507676482200623, -3.1776082515716553, -1.671351432800293, -0.11263690143823624, -0.6860203742980957, -0.3193381428718567, -0.07543419301509857, 0.5167030096054077, 0.2749791443347931, -1.7203227281570435, 1.7025448083877563, -1.772176742553711, -0.9260359406471252, -0.4531424641609192, -1.9338828325271606, -1.1955078840255737, 0.9044212102890015, 1.5016592741012573, 0.25700488686561584, -0.41659194231033325, 0.6146079897880554, 0.4571688771247864, -1.1624993085861206, -1.5358452796936035, -1.0457361936569214, 2.0820322036743164, -0.3246244490146637, 0.302594929933548, -0.1746618002653122, 1.0470980405807495, 0.3669365346431732, -0.9910060167312622, -1.792522668838501, -0.48721206188201904, 1.185524582862854, -1.5023528337478638, 0.8238177299499512, 0.008935061283409595, 1.2548670768737793, -1.08406662940979, 0.16328543424606323, 1.0830148458480835, 0.8035480976104736, -0.1598632037639618, -1.8475494384765625, -1.431685447692871, 0.035594675689935684, 1.0318493843078613, 0.00445921067148447, -0.3239094913005829, -1.681878685951233, 0.9375402331352234, -1.7759565114974976, 0.353659451007843, 0.8297300934791565, 0.8140329122543335, -2.6638426780700684, 0.09584594517946243, 0.10636056214570999, -1.2588773965835571, 0.8281317949295044, 0.3279889225959778, 0.4007571339607239, -0.9072612524032593, 0.5838682055473328, -0.5983701348304749, -1.8448847532272339, -1.2870454788208008, 0.3341851532459259, 1.2817479372024536, -0.12361498922109604, -0.7375341653823853, -0.6815366744995117, -0.12873311340808868, -1.1228001117706299, -0.602656900882721, 0.4719933867454529, 1.3669008016586304, -1.4791077375411987, -0.3001680076122284, 1.453357458114624, -0.9761927127838135, -0.4357980191707611, 1.1779166460037231, 0.19498591125011444, 0.10096585750579834, -1.7552330493927002, 0.19433054327964783, 0.11298371106386185, -1.0863399505615234, 1.1934268474578857, 0.6087483763694763, 0.8609811067581177, 1.976198434829712, -0.3452315330505371, -0.19548583030700684, -0.38025209307670593, -0.5378693342208862, 0.8938164114952087, -0.935303807258606, -0.41935864090919495, 0.7329735159873962, 2.7944326400756836, 1.142046570777893, 0.8812901377677917, -0.2847093641757965, 1.3453854322433472, 0.11766045540571213, -1.0715140104293823, -1.9530705213546753, 0.8709448575973511, 0.2903265357017517, 0.588038980960846, -0.8739902973175049, 1.2353123426437378, -0.15907080471515656, 0.1294855773448944, 2.0064148902893066, -1.1483967304229736, 1.7659443616867065, 0.5145386457443237, -1.492893099784851, -0.3204679787158966, -1.5651171207427979, 0.0076639698818326, 0.4339861571788788, -0.08278205245733261, 1.1759992837905884, 0.5109658241271973, -0.5375939607620239, -0.8818647265434265, 0.76951003074646, 0.8579476475715637, 0.04382595047354698, 0.5077655911445618, 0.0451316237449646, 0.12400210648775101, -0.7379239201545715, 1.977941632270813, 1.0882277488708496, 0.24891918897628784, 1.7714110612869263, -1.1795765161514282, 0.6106144189834595, -0.002944842679426074, 0.7225168943405151, 0.7837274074554443, -0.7672501802444458, 0.12269490957260132, 1.5094531774520874, 0.9709616303443909, -0.8319082260131836, -0.7837597131729126, -1.3193000555038452, -0.11120030283927917, -1.3731467723846436, -0.20334923267364502, 0.32697969675064087, -0.8004963397979736, 0.019776280969381332, -0.6556915044784546, 0.3381079137325287, -0.615659236907959, 0.2141478955745697, 1.3616876602172852, 0.7749243974685669, -0.4335425794124603, 0.6504859328269958, -1.1792393922805786, 1.4048476219177246, -0.701944887638092, -0.9289298057556152, -0.43685275316238403, -0.44716155529022217, 1.452010989189148, 1.5135681629180908, 0.1736307442188263, 1.6054339408874512, -0.3539542257785797, -0.34317746758461, 0.8655673265457153, -1.4506428241729736, 1.111149549484253, -0.3459749221801758, 0.18082064390182495, -0.851169764995575, -0.5147474408149719, 1.1750813722610474, 1.6328884363174438, 0.5011611580848694, -0.6647047996520996, 0.5678163170814514, 0.8439372777938843, 0.06615497916936874, 0.4181831181049347, -1.4555367231369019, 0.2676609754562378, 0.36799368262290955, -0.29110610485076904, 0.8466300964355469, 2.5597829818725586, -1.4785913228988647, -0.23028594255447388, -0.13687701523303986, 0.5864365100860596, 1.118565320968628, -1.2343955039978027, 0.984988808631897, -0.13232475519180298, -1.3720014095306396, 0.5690196752548218, -1.6758259534835815, 1.84305739402771, -0.15968573093414307, 1.1966253519058228, -0.4170120656490326, 0.07686250656843185, -1.0666218996047974, 0.5466724634170532, -0.9384596943855286, -0.3550938069820404, -0.4739406108856201, -2.573639154434204, -0.8192915320396423, -0.2797199785709381, -0.7977107167243958, -0.9432611465454102, 0.22143492102622986, 0.7628119587898254, 0.8190849423408508, -1.5254675149917603, 0.5621525645256042, -1.0534868240356445, -0.3727622330188751, -0.32234787940979004, -0.10872161388397217, -0.7038071751594543, 0.5199392437934875, -0.34645846486091614, 0.5028415322303772, -0.2648313641548157, -0.5253604650497437, 0.46496397256851196, -1.1267741918563843, -0.7089009881019592, 0.6447544097900391, 1.0338313579559326, 0.541969895362854, -0.3169706463813782, 1.7801034450531006, 0.08436962217092514, 1.0361711978912354, 0.532660186290741, -0.08266304433345795, 0.942861795425415, 0.11452917009592056, -0.1687629669904709, 0.3986680507659912, -1.6628077030181885, -0.4939374625682831, -0.8466943502426147, 1.1498327255249023, -0.4660300314426422, -2.0136184692382812, -0.5109008550643921, 1.0182965993881226, 1.5865259170532227, 0.586531937122345, -0.33813580870628357, -1.2559242248535156, -0.057446036487817764, -0.8379485607147217, 0.8326781392097473, 0.081366628408432, 1.1811031103134155, 0.9236529469490051, -1.1769417524337769, -0.6637954711914062, 0.11944485455751419, -1.208356499671936, 0.09946376830339432, 0.5173977017402649, -0.850464403629303, 0.4916219413280487, -0.12705011665821075, -1.3592716455459595, -2.9203171730041504, -1.1389676332473755, -2.1874516010284424, 0.8139209747314453, -0.5302308797836304, -0.12230803817510605, 0.5013212561607361, -1.9514782428741455, -1.1344037055969238, 0.7596339583396912, -0.7535894513130188, -1.448496699333191, 0.07268331944942474, 0.333866149187088, 0.37068575620651245, -0.7751108407974243, -0.0003952556289732456, -0.811345100402832, -0.21362118422985077, 0.9689565300941467, -0.3246266543865204, -0.9187684655189514, -0.685658872127533, 0.30917948484420776, 0.33382993936538696, -0.4982987642288208, 1.662581205368042, 0.278516560792923, 0.4425816833972931, -0.13296908140182495, 0.6479310393333435, -0.0529002882540226, -0.38751086592674255, 1.1817266941070557, -0.5018897652626038, -0.6091736555099487, -1.4732022285461426, -0.5302269458770752, -0.09133005142211914, -0.8271602988243103, -0.4635322391986847, -2.921111822128296, -1.3241157531738281, 0.9010345339775085, -0.9846408367156982, 0.4945867955684662, 2.3547191619873047, -1.6087321043014526, 0.8223127126693726, -0.5697874426841736, 0.09336617588996887, -1.0861045122146606, 0.7824730277061462, 0.37824544310569763, -0.2815837860107422, 1.6685367822647095, -0.07894215732812881, 2.372061252593994, 0.3889276683330536, 1.8960381746292114, 1.1551254987716675, 0.26256805658340454, 1.2317593097686768, 0.0402863435447216, 1.3213037252426147, -0.37595728039741516, 0.3184982240200043, 0.19835732877254486, -0.5453971028327942, 1.1512271165847778, 1.6616134643554688, -1.8420062065124512, 1.241509199142456, 0.030966470018029213, 1.0643973350524902, 1.7615762948989868, -1.819779872894287, 1.0389211177825928, -2.383286714553833, 0.2180950939655304, 0.07421448826789856, -0.960081934928894, -0.6843518018722534, -1.0819181203842163, 0.28332364559173584, -0.17616190016269684, -0.11402645707130432, 1.320157527923584, -1.098530888557434, -0.7898565530776978, -2.107264518737793, -0.12331880629062653, -0.5366845726966858, -0.10999814420938492, 0.7520489692687988, 0.3810001611709595, -1.5106585025787354, -0.39921823143959045, 1.7016068696975708, 1.2689390182495117, 1.0233752727508545, 0.6694860458374023, -0.5061866044998169, -2.2656586170196533, -0.2616459131240845, 1.7587432861328125, -0.5805324912071228, -0.7966724038124084, -1.912789225578308, -1.2362325191497803, -0.07602415978908539, 0.6948450207710266, -0.27947551012039185, 0.17600612342357635, 1.0875242948532104, -1.5584840774536133, -0.40408122539520264, -0.16169720888137817, -0.06591479480266571, -0.3635531961917877, -0.5270986557006836, -0.7537804841995239, -0.6528795957565308, -0.2176632285118103, -0.19019366800785065, 1.450907588005066, 1.0323104858398438, -0.3366181254386902, 1.860486388206482, -0.8820982575416565, -0.9159572720527649, -0.6207069158554077, 2.600374937057495, -1.111172080039978, 0.4040037989616394, 0.23864127695560455, -0.1481224000453949, 0.5973042249679565, 2.0694291591644287, -0.37144529819488525, 0.8797740340232849, 0.023359114304184914, -1.4966403245925903, 0.5286282300949097, 0.6137606501579285, -0.19072158634662628, -1.4985226392745972, 0.8454630970954895, 1.165320634841919, 0.6917175054550171, -0.18589811027050018, 0.7890672087669373, 0.06684418022632599, 1.2832906246185303, -0.6420410871505737, -1.7975871562957764, 0.1258394569158554, 0.6536111235618591, -0.35721608996391296, 1.2706615924835205, -1.0054246187210083, -0.07629838585853577, -0.06577669829130173, -1.7270833253860474, -0.04092355817556381, -0.6328349113464355, -0.6065787672996521, 1.688474416732788, 0.576358437538147, -0.4187917709350586, 1.2125728130340576, -0.42853495478630066, -0.14392635226249695, -0.1286495327949524, -0.7842934131622314, -1.659194827079773, 1.8169487714767456, 1.2888267040252686, 0.23422826826572418, -0.35508033633232117, -0.03737076744437218, 1.0709213018417358, -0.03196949511766434, 0.17071977257728577, 1.5207101106643677, 1.1111096143722534, -0.7918158173561096, -0.18036364018917084, -0.5051029324531555, -1.502226710319519, -1.1644835472106934, 1.8168798685073853, 1.179670810699463, -0.6109125018119812, 0.6164442300796509, 1.0227909088134766, -0.6399275660514832, -0.654721736907959, -0.5786328911781311, -2.0469233989715576, -0.9689775109291077, -0.3884536325931549, 0.38439226150512695, 0.7542991638183594, -1.2890992164611816, 1.533092737197876, 0.4922764003276825, -0.9816428422927856, 1.1343399286270142, -1.535549283027649, 0.037568770349025726, 0.2791723310947418, -1.1715887784957886, -1.0321037769317627, -1.0469671487808228, 0.3789631426334381, 0.22656257450580597, -0.7445027232170105, -2.1670351028442383, -1.2218152284622192, -0.45290815830230713, -0.7798267602920532, 1.1079609394073486, 0.15762637555599213, -0.4265814423561096, -0.6954153776168823, -0.37051740288734436, 0.24942660331726074, -0.3404751420021057, 0.8362373113632202, 1.4374892711639404, -0.07607073336839676, -0.541319727897644, -1.1545428037643433, -1.1364495754241943, -0.18141895532608032, -0.7559794783592224, -1.3136401176452637, 0.937781572341919, 0.006289815995842218, 0.4618062376976013, -0.6691875457763672, -0.4228794574737549, 0.41419777274131775, -1.391631841659546, 0.3947746753692627, -1.055403709411621, 0.18132010102272034, -0.12851019203662872, -0.9374532103538513, -0.7127143740653992, 1.4637285470962524, -0.15680958330631256, -0.9339806437492371, 0.10902088135480881, -0.6482300162315369, -0.523124635219574, 1.1524423360824585, -0.0618804469704628, 1.8825230598449707, 0.757034957408905, -0.817058265209198, -0.31850665807724, 0.03201699256896973, -1.3047846555709839, -2.007514238357544, -1.0914201736450195, -0.7668169140815735, 1.269850730895996, -0.5629530549049377, -0.12889868021011353, 1.5363523960113525, 0.8172993063926697, 1.822935700416565, -0.9188526272773743, 0.26664450764656067, 0.2531575560569763, 2.3326363563537598, -2.780113935470581, 0.8037134408950806, -0.5408704280853271, 0.5085911154747009, 0.8796284198760986, 0.036008745431900024, -1.4250940084457397, -2.8111777305603027, 0.3036048114299774, 0.1748548299074173, -1.6648859977722168, -0.3916419744491577, 0.19384059309959412, -0.0745910257101059, 0.05134597048163414, -0.08246659487485886, -0.3047301769256592, 2.1073532104492188, -0.5389047861099243, -0.45162928104400635, -1.1852318048477173, 3.076910972595215, 1.0799778699874878, -1.3140132427215576, 1.231731653213501, -0.5052006244659424, 0.30534854531288147, 1.0606513023376465, 0.7211211919784546, -0.8544769287109375, -1.1849428415298462, -0.013168714009225368, 1.2023282051086426, 2.507770299911499, -0.6505138874053955, 1.6535733938217163, 0.647225022315979, -1.3102385997772217, 0.8952365517616272, -0.7733595371246338, -1.223403811454773, 0.3650515079498291, -0.9566486477851868, -0.033594775944948196, 0.15077419579029083, -0.40941882133483887, 1.223623275756836, -0.7458676695823669, 0.21393191814422607, 0.49125781655311584, 0.5043705701828003, 0.31087157130241394, -1.0945844650268555, 0.22353841364383698, 0.3868809640407562, 1.389139175415039, -0.06503564864397049, -1.0777603387832642, 0.7237105965614319, -0.27775809168815613, -0.20140214264392853, -1.4662344455718994, 0.5764631628990173, 0.29407376050949097, 0.2862859070301056, -0.5818091034889221, 0.25595736503601074, 1.3754409551620483, -0.1681620180606842, -0.4391224980354309, 0.19712796807289124, -0.9160479307174683, -0.058271851390600204, 0.5263011455535889, 0.10892728716135025, -0.7945212125778198, -1.052973985671997, 0.19418488442897797, -1.1453659534454346, -0.573854386806488, 1.084726333618164, -0.4991300106048584, -0.10610929876565933, -1.486525535583496, 1.994015097618103, -0.07252782583236694, -1.4286576509475708, 0.33842188119888306, 0.8682960867881775, -2.3913536071777344, 0.691707968711853, -0.7427387237548828, 0.03991863131523132, 0.2953832745552063, 0.35133665800094604, 2.007801055908203, 0.8632338643074036, -0.41811731457710266, 0.4918749928474426, 0.31995442509651184, -0.4301500916481018, 0.00193809496704489, 0.37870553135871887, -0.6673427820205688, -0.6332830190658569, -0.797904908657074, 0.394282728433609, 0.5832744836807251, -0.11261127889156342, 0.4468102753162384, -0.4348832666873932, 2.2084732055664062, 0.8543416261672974, 0.11115501821041107, 0.4850514829158783, -1.2466599941253662, -0.4038454294204712, -0.6225072741508484, -1.4916832447052002, 1.2339143753051758, -0.37604233622550964, 0.7588857412338257, 0.37358397245407104, 0.18307346105575562, 0.4165549576282501, 0.4969973862171173, 0.5478288531303406, -0.3056498169898987, -0.33068978786468506, -0.34455546736717224, -1.4759279489517212, 0.5687344074249268], [-2.7227606773376465, -1.0292036533355713, -0.023315303027629852, 1.6738014221191406, -0.3139534294605255, 1.2229443788528442, 1.3163971900939941, -0.8614870309829712, 0.6174502372741699, -0.3199927806854248, -2.2930684089660645, -0.8492197394371033, -0.5569801926612854, 1.668636441230774, 0.8602545261383057, 0.40820974111557007, 0.5771929025650024, -1.2921302318572998, 0.9288654923439026, -1.0166535377502441, -0.7575616836547852, 1.632414698600769, 0.26614928245544434, 0.7779139280319214, 0.12614542245864868, -2.2944459915161133, 1.3080023527145386, 0.19117659330368042, 0.5858715176582336, 0.7729712724685669, -0.8072074055671692, -0.9482409954071045, 0.6571775674819946, -0.23400326073169708, 0.20560725033283234, -1.2189736366271973, 0.2319553643465042, 0.7836815714836121, -1.5328136682510376, 0.845302164554596, 0.4254012703895569, 0.6819725632667542, -0.4092203676700592, -0.8713724613189697, -1.2607942819595337, 0.7866623997688293, -0.9048619270324707, 0.5961517095565796, -0.4413478970527649, -0.23022934794425964, 0.8645191788673401, 1.401930332183838, -0.27624332904815674, 0.0005745426169596612, -1.6856327056884766, -1.115919828414917, 0.22100716829299927, -0.1356222927570343, 0.25252217054367065, 0.7157954573631287, -2.22487473487854, 0.6198946237564087, 0.5997865796089172, 0.5518919229507446, 0.15302766859531403, 0.020006483420729637, -1.3878283500671387, 0.9182237982749939, 0.9147226810455322, -0.8366503715515137, 0.8684065937995911, -0.4907737076282501, 1.9108259677886963, -0.13613449037075043, 0.93109130859375, 0.42932045459747314, -1.2299139499664307, 0.3702986240386963, 0.4959287941455841, 0.5919259786605835, -1.8386352062225342, 0.04083086922764778, -0.737485945224762, 0.056592777371406555, -1.93169367313385, 0.8109848499298096, -0.5834439396858215, -0.3389715552330017, 0.6706286072731018, 0.08076438307762146, -0.334126740694046, -0.5495507717132568, 4.634023666381836, -0.44665318727493286, -0.02195172756910324, 0.6360454559326172, 0.9990792870521545, -0.7994779944419861, 0.4074464738368988, 0.28406181931495667, -0.20648466050624847, 0.8590795993804932, 0.10922686010599136, -0.1438581645488739, 1.0613365173339844, -0.7300959229469299, 1.160340666770935, -0.9403575658798218, 0.2906079888343811, 1.0146390199661255, 0.37431877851486206, -1.2906138896942139, -0.04214588552713394, -0.9776880741119385, 0.7031599283218384, 0.11986745148897171, 0.1278533637523651, -1.261687159538269, -2.6381306648254395, 0.4660962224006653, 0.12460710108280182, -0.3614949584007263, -1.3272509574890137, 2.0883283615112305, -1.093538522720337, -0.7042180299758911, 2.0430212020874023, -0.5229055285453796, 1.328773856163025, 0.3673589825630188, -0.5845671892166138, -0.3437788784503937, -0.06916676461696625, -1.4491547346115112, -0.8900583386421204, -0.7716444134712219, -0.29117968678474426, 0.2946338653564453, 0.12519674003124237, -0.3799659311771393, -1.5446586608886719, -0.762988269329071, -0.2730400264263153, 1.3143573999404907, 1.6599565744400024, 0.5753102898597717, -0.5930132269859314, -1.1205964088439941, -0.9097334146499634, -0.4679866135120392, -1.294802188873291, 1.034930944442749, 0.7763695120811462, 1.4147382974624634, -0.27716803550720215, 0.25118711590766907, -1.5996373891830444, -1.2143101692199707, 1.3900151252746582, 0.7943230271339417, -2.2692463397979736, 0.028870245441794395, 0.45817288756370544, -0.8119927644729614, -0.667029857635498, 0.1762641966342926, 2.0529584884643555, -0.2616015672683716, -1.7584551572799683, 0.7605370283126831, -1.0040876865386963, 1.597605586051941, 0.24288047850131989, 0.45697125792503357, 0.5778613686561584, -1.3543273210525513, -2.2567808628082275, -1.063951015472412, 0.6504198312759399, -1.496948480606079, 0.4896180331707001, 0.03287380933761597, 0.4121655523777008, -0.2047494351863861, 0.469868928194046, 0.8619787096977234, 1.3157583475112915, 1.5066215991973877, -0.4413490295410156, -0.20461896061897278, -0.9167106747627258, -0.3155553638935089, -0.6704413890838623, -0.3200114965438843, -0.24983437359333038, 1.0052666664123535, -1.1570473909378052, 0.18554425239562988, -0.168961301445961, 2.1488373279571533, -0.19452151656150818, -0.6675189137458801, 0.3151605427265167, -1.2512049674987793, 0.2995123267173767, -0.8560616374015808, -0.36950618028640747, -0.06192828342318535, -0.32100772857666016, 0.5232065916061401, 0.48200368881225586, 1.3872443437576294, -0.11193586140871048, 2.3152620792388916, -0.689204216003418, 1.1451680660247803, -0.867191731929779, 0.8482518792152405, -0.8068357110023499, -0.5214694738388062, -0.8687017560005188, -0.4427163898944855, 0.77203768491745, -1.1907126903533936, -0.8761853575706482, -0.5385942459106445, 0.8745616674423218, -1.2245227098464966, 1.7249804735183716, -0.016602778807282448, 0.07566847652196884, 2.4519522190093994, 0.657375156879425, 0.13552185893058777, 1.0663683414459229, 0.2271069437265396, -0.5779176354408264, 2.1205668449401855, 1.1802425384521484, 1.367367148399353, -0.5830495953559875, 0.09038874506950378, 0.6527442336082458, -1.1201748847961426, -0.1264747679233551, -0.5189183354377747, 0.018909377977252007, 0.8330159783363342, -0.0653066635131836, -0.265140563249588, 0.35837554931640625, 1.5443778038024902, -2.491461992263794, -1.2791780233383179, -0.2665565609931946, 0.3380739390850067, 2.0748932361602783, 0.47315433621406555, 0.4921765923500061, 0.36228251457214355, -1.691801905632019, -0.3835684657096863, 1.5565458536148071, -0.7512335777282715, 0.0334315150976181, -0.6985740661621094, 0.1478373408317566, 0.24936631321907043, -1.3643982410430908, 0.3673311173915863, 0.27464374899864197, 0.5173417329788208, -0.3997584283351898, -0.9316473603248596, 0.3718852996826172, -0.20277731120586395, 2.0718066692352295, 0.9470790028572083, 1.7642852067947388, -0.7067104578018188, 0.18005932867527008, 0.5996588468551636, -0.026866812258958817, -1.0086162090301514, -0.8641011714935303, 0.21747541427612305, -0.2341698408126831, 1.0083695650100708, -0.43765339255332947, 0.2570406198501587, 0.8221216201782227, -0.7412750124931335, 0.2828410863876343, -0.7734120488166809, -0.4204770624637604, 0.14375102519989014, 0.398710697889328, 1.4423657655715942, -1.1923383474349976, -0.12971942126750946, -0.8420670032501221, 0.39492857456207275, -0.8270489573478699, 0.42681294679641724, -0.921576738357544, 0.8561466336250305, -0.16756165027618408, 0.6120347380638123, -1.193958044052124, 1.0383208990097046, 0.27472588419914246, -0.804262101650238, 0.6298775672912598, 2.1803808212280273, -0.30012911558151245, -0.34646475315093994, -1.1547935009002686, -0.1528288871049881, -0.22157584130764008, 2.0631444454193115, 0.6144471764564514, -0.09214077889919281, -0.3872852325439453, -0.0995420515537262, 1.5691406726837158, -0.44185197353363037, 1.2357033491134644, 1.3870677947998047, 1.6327534914016724, 0.2739224135875702, -0.05085050314664841, -0.4555671215057373, 0.6200392842292786, -1.1369301080703735, -1.8186753988265991, 1.214113473892212, -1.1569910049438477, 2.0350961685180664, 0.01441869419068098, -1.071959137916565, -0.25695937871932983, -0.7593346834182739, 1.6541515588760376, -0.7250744700431824, -0.24763548374176025, 0.6221303343772888, 0.3651128113269806, -0.1791389137506485, -0.7087651491165161, 0.8660090565681458, -0.7582617402076721, -0.4646679759025574, 0.5518572926521301, -1.1261564493179321, 1.0815008878707886, 1.222825050354004, -2.102273941040039, 0.2133677750825882, 0.9216191172599792, 0.33279722929000854, 0.6633949875831604, -0.5339974164962769, 0.6246325969696045, 2.728041648864746, 0.33103621006011963, 1.1159595251083374, -0.7429810166358948, 0.7232351303100586, -0.4655117392539978, -1.0252487659454346, -0.7008944153785706, 0.632959246635437, 0.374831885099411, -1.2482848167419434, 0.6999572515487671, -1.247778296470642, 2.276198387145996, 0.32678765058517456, -1.0702896118164062, -2.371957540512085, -0.771732747554779, -1.251754641532898, 0.9263911247253418, -1.8012770414352417, -0.5692501068115234, -0.48396944999694824, -1.7243214845657349, 0.5157638788223267, -2.2269654273986816, 0.2640392780303955, 2.1181766986846924, 2.1015853881835938, 0.9866760969161987, 0.6614812612533569, -2.180676221847534, 0.1896638572216034, 0.9043282270431519, -1.1969659328460693, 0.6111224293708801, -0.5001396536827087, 0.5998809933662415, -1.0475986003875732, 0.48629623651504517, 0.553353488445282, 0.8972722291946411, 0.12349561601877213, 1.4156992435455322, -2.1575231552124023, -1.5248939990997314, 0.8801943063735962, -0.5003570318222046, 0.07522052526473999, -0.581929624080658, -0.32795923948287964, 0.8595080971717834, 2.9255692958831787, 0.6384819149971008, -0.8538263440132141, -0.565210223197937, -0.28996121883392334, 0.21319317817687988, 1.991492509841919, -0.4231635332107544, 0.33863574266433716, -0.20486930012702942, 1.0695863962173462, 1.2536870241165161, -0.3987995684146881, -0.5043655037879944, -0.6884558796882629, -1.265270709991455, 0.23284460604190826, 0.5559467673301697, 0.22889859974384308, -1.4213683605194092, -0.42058438062667847, 1.5521478652954102, 0.5444055795669556, -1.0215891599655151, -1.4769306182861328, 0.3963247239589691, -2.4302778244018555, 0.3875267207622528, -0.005547236185520887, -1.2889829874038696, -2.694927215576172, 0.1776149719953537, -0.7699674367904663, 0.6785645484924316, -0.10313498973846436, 1.0595824718475342, 0.26253455877304077, -0.816239058971405, -0.4942414164543152, 2.1165919303894043, -2.708070993423462, 0.3233283758163452, 0.416191041469574, -0.8391368985176086, -0.6080850958824158, -1.1850452423095703, -0.421756386756897, -0.20912010967731476, 1.194042444229126, 1.1183154582977295, -0.04766411706805229, -0.2110840231180191, 0.6521152853965759, -1.7175055742263794, 0.07239347696304321, -1.546544075012207, -1.4216904640197754, 0.5157383680343628, -1.6532188653945923, -0.9752432703971863, -0.8752244114875793, 1.4751231670379639, -0.10397195816040039, -0.003892164444550872, 0.27310988306999207, -0.23911920189857483, -1.4501209259033203, -0.25152018666267395, -2.680997848510742, -0.826276421546936, -0.22367773950099945, -0.10757917910814285, -0.06188533827662468, 1.3550636768341064, -0.9276511073112488, 1.653496265411377, -1.5544294118881226, 0.6477795839309692, 0.48205986618995667, 1.0353304147720337, 1.9957761764526367, -1.048343539237976, -0.43832021951675415, 1.3555915355682373, 0.10598142445087433, 0.7236152291297913, 1.6893781423568726, -1.0246175527572632, -0.40017393231391907, -0.7976475358009338, 0.6893362998962402, 1.184333324432373, -0.18304097652435303, -0.4928715229034424, 1.8823121786117554, 0.8127849698066711, -0.10824890434741974, -1.595950961112976, 1.8148524761199951, -1.4691132307052612, -0.039688028395175934, 0.5902679562568665, 0.8756008148193359, 0.10123438388109207, 0.7669674754142761, -0.6869872212409973, -0.18271343410015106, 1.1828725337982178, 2.5413458347320557, -1.0407700538635254, 1.8380146026611328, 0.47298839688301086, -0.972474217414856, 0.41644981503486633, -0.26185497641563416, -1.4442455768585205, -0.20665015280246735, 1.217531442642212, -2.0350446701049805, 0.4972604811191559, 1.6297239065170288, 1.5969558954238892, -2.2062833309173584, 0.37036287784576416, -0.2754342555999756, -0.6949643492698669, 0.5824670195579529, -0.41672948002815247, 0.7940728068351746, -0.41558268666267395, -0.34919512271881104, -1.9977127313613892, -0.19901491701602936, 1.12142813205719, -0.3044309616088867, -0.6891337633132935, -0.19973845779895782, 0.33610546588897705, -0.8593541979789734, -0.7651568055152893, 0.6012712121009827, -0.15074288845062256, -1.2938450574874878, -0.02438855543732643, -2.0760068893432617, -0.7100158929824829, 0.033174771815538406, -0.35848331451416016, 0.8119816780090332, -1.8855197429656982, -1.4468258619308472, -0.725645124912262, 0.6350592374801636, 2.130800485610962, 0.1378871202468872, 0.11662771552801132, -0.4832378625869751, -0.14255304634571075, -1.1341042518615723, -0.08011836558580399, -0.5248538851737976, 0.17326465249061584, 0.1524256318807602, 1.253684401512146, 1.821582555770874, -0.500525951385498, -0.47146522998809814, 0.0891847237944603, 2.3887779712677, -1.27768075466156, 0.8735455870628357, -0.5735164880752563, -1.5233163833618164, -1.1036278009414673, 1.0156128406524658, 0.8600021004676819, 0.4182363748550415, 1.1737128496170044, -0.6277602910995483, -0.3979051113128662, -2.0216450691223145, 0.6666528582572937, -0.5007513761520386, 1.5308988094329834, 1.4431333541870117, 1.2409371137619019, 0.32323339581489563, 0.25607573986053467, -0.3035685122013092, -0.8762891292572021, 0.7392137050628662, 1.8620853424072266, 0.16311602294445038, -0.05814072862267494, -0.5433464050292969, 0.32548487186431885, -0.4216265082359314, 1.518674612045288, 0.6230367422103882, -0.2901573181152344, 0.0207520704716444, -1.0176316499710083, -2.875303030014038, -1.5430079698562622, -0.38934841752052307, 0.837214469909668, -0.4233105480670929, -0.10439097136259079, -0.6722438931465149, 0.9392926096916199, 0.01934574544429779, -1.6749482154846191, -0.35699525475502014, 0.5942460894584656, 0.6622098088264465, -0.47058162093162537, 0.9555128216743469, -0.4622097611427307, 0.2797321677207947, 0.2778683602809906, 0.3058886229991913, -0.7693178653717041, -0.44754910469055176, -1.8299314975738525, -1.490344762802124, 0.8526771068572998, 0.7862006425857544, -0.3390584886074066, 0.23242345452308655, 0.4418661296367645, 0.7090669274330139, -2.484917640686035, 0.7829573154449463, 0.7435773611068726, 1.8913023471832275, -0.8293231129646301, 0.1832379251718521, 1.54668390750885, 0.6254726052284241, 1.210516095161438, -0.47978299856185913, 0.6300084590911865, 1.0714651346206665, 0.9643784761428833, 1.256564736366272, -1.1090259552001953, 0.18199992179870605, 1.3604247570037842, -1.4319652318954468, 2.9359798431396484, 0.6673402190208435, 2.226605176925659, -1.3989118337631226, 0.24730269610881805, 1.0640801191329956, 0.9048847556114197, -1.4759784936904907, -1.139830231666565, 1.1639732122421265, -1.270369291305542, 0.31032729148864746, -0.9340142607688904, 0.6691868305206299, 0.5220671892166138, -1.5663267374038696, -1.1863734722137451, 0.6242222189903259, 0.8010029196739197, 1.3823487758636475, -0.6340475082397461, -1.3238166570663452, -0.5542059540748596, 1.5404802560806274, 0.5603281855583191, 2.7957570552825928, -1.4738023281097412, 0.6780071258544922, 0.5536016821861267, 1.4657033681869507, -0.4222621023654938, -0.18431447446346283, -0.5459280014038086, -0.3479984700679779, -0.15858635306358337, -0.6648122668266296, 1.028429388999939, -0.2475285679101944, -0.1970003843307495, -1.6652547121047974, -0.39950263500213623, -1.1800487041473389, 0.9942514896392822, -1.704121708869934, 1.1503981351852417, -0.1136770248413086, 0.4630855321884155, 2.3615102767944336, -1.5797579288482666, -0.6626310348510742, 1.8105559349060059, 0.09989558905363083, 0.007206783164292574, 1.5886445045471191, -0.351608544588089, 0.152265727519989, 0.6271718144416809, 0.931988537311554, 0.8265197277069092, -1.752936601638794, -0.5963298678398132, 0.6877357363700867, 2.988159656524658, 0.5160593390464783, -1.5469422340393066, -1.5987348556518555, -1.617607593536377, -0.8079420924186707, 1.1433120965957642, -1.0053750276565552, 0.08226783573627472, -0.03346792981028557, 0.47212716937065125, 0.2997094690799713, 0.6007587313652039, -3.235123634338379, -0.7202691435813904, 0.3808465898036957, -0.03410317748785019, -0.4920089840888977, 0.7406008839607239, -0.36139971017837524, -0.5769290328025818, 1.0377570390701294, -0.7168003916740417, -0.45221972465515137, -0.6865422129631042, -0.28681570291519165, 0.2388399839401245, 0.7467449307441711, 0.8402349352836609, 0.2637649178504944, 0.19016851484775543, 1.2857131958007812, 0.280256450176239, 0.48918214440345764, -1.3892436027526855, 0.008529536426067352, 0.5614109635353088, 0.7124326229095459, 0.2803488075733185, 1.2370669841766357, 0.2852148115634918, -1.870903730392456, 0.546405553817749, -0.12195337563753128, 2.578643321990967, 0.3224167227745056, 1.4471532106399536, 0.24956144392490387, -1.0149272680282593, -1.3443562984466553, 0.747860312461853, 0.4391266107559204, 0.7806945443153381, -0.602708637714386, 0.7873267531394958, 0.05004650726914406, 2.176373243331909, 0.4894464910030365, -0.7744395732879639, -0.7463333606719971, 0.2610985040664673, -0.2668079733848572, 1.5497945547103882, 2.118211030960083, 1.277753472328186, -0.6390503644943237, -0.1780068278312683, 1.5557754039764404, 1.8610289096832275, -0.11996341496706009, 1.5984619855880737, -1.6181429624557495, 0.06528228521347046, 0.882710874080658, -1.0767319202423096, -0.4882279634475708, 0.13518285751342773, 0.06924153119325638, 1.629353642463684, -0.09945040941238403, 0.8736247420310974, -1.2078732252120972, -0.4611090123653412, 1.289947271347046, -0.5321751832962036, -1.0220369100570679, -0.659759521484375, -0.8658455610275269, -0.7703933715820312, -0.10622648894786835, -1.1763219833374023, -0.35590967535972595, 0.9338560700416565, 1.4893490076065063, -0.6257715225219727, -1.143998622894287, 0.6942743062973022, -0.03164536878466606, 0.5305818915367126, -0.35252314805984497, 1.1420847177505493, 0.1246563270688057, -2.4502062797546387, -1.3955849409103394, 0.10267519950866699, 0.6564301252365112, 0.18433986604213715, -0.829521656036377, -2.592294454574585, 0.3725802004337311, 0.24916642904281616, 0.5643821358680725, -1.4162794351577759, 1.9049444198608398, -0.43871650099754333, -0.08781155198812485, -0.03368060290813446, -0.25765231251716614, -0.41332247853279114, 1.5564128160476685, -1.1444050073623657, -0.5493795871734619, 0.6539479494094849, 0.9119461178779602, 0.2832246422767639, 0.9633790254592896, 0.16877180337905884, 0.012965377420186996, 0.059033870697021484, -0.03187936916947365, -1.6036183834075928, 0.5676788687705994, -0.18908192217350006, 2.1729111671447754, 0.4801334738731384, -0.15895964205265045, -0.7729568481445312, -1.5738868713378906, -0.014460073783993721, 0.165123850107193, -1.2177293300628662, 1.009535551071167, 1.7078654766082764, -0.532307505607605, -1.4455997943878174, 1.3820348978042603, -1.462777018547058, -0.035475652664899826, 0.2511657178401947, 0.7723566889762878, -0.3980949819087982, 1.1842119693756104, 0.3799329698085785, -2.602654218673706, -0.8894962072372437, 0.8009719848632812, 0.7358248829841614, -0.4022143483161926, 0.14613491296768188, -0.6146612763404846, 0.6398969292640686, -0.2505817711353302, -0.015051433816552162, -0.5335765480995178, -0.30532923340797424, -1.9804346561431885, 1.4155733585357666, 2.1547043323516846, 1.3103446960449219, -1.3314685821533203, -0.24261189997196198, 0.3750498294830322, -2.5065014362335205, -0.6576970815658569, 0.5702275037765503, -0.3933815658092499, 0.7169720530509949, 1.0022388696670532, -0.0999520868062973, 0.6850923299789429, 1.603772759437561, 1.2457046508789062, 0.5525758266448975, -1.4326132535934448, -1.3907593488693237, -1.923262596130371, -0.12029757350683212, -0.6575765609741211, 1.615273118019104, -1.7187952995300293, 1.6812357902526855, 0.16362453997135162, 1.4480602741241455, 0.7023919224739075, 1.1846179962158203, -0.2961832284927368, 0.6912820339202881, 0.5308083295822144, 0.6631272435188293, -1.3717682361602783, 0.09780099242925644, -1.0169178247451782, -1.5312131643295288, -0.07914220541715622, -0.18379317224025726, 0.30551812052726746, -0.16018743813037872, 0.9557313323020935, 0.34934255480766296, -0.005201342049986124, 0.1378915160894394, 0.5401454567909241, 0.11258996278047562, -0.023021703585982323, 1.1860684156417847, 1.705946922302246, 0.06694352626800537, 0.9024941921234131, 0.4484248757362366, -1.508130431175232, 1.3484070301055908, -1.1194183826446533, 2.2624707221984863, -0.9357264637947083, 0.1611568033695221, 1.2331141233444214, -0.2665214240550995, 1.1711231470108032, 0.4497346580028534, 0.5652187466621399, -0.5916915535926819, 1.0922625064849854, 0.1784549206495285, -0.4068394601345062, 0.41953858733177185, -1.9234362840652466, -0.7836518883705139, -0.8389675617218018, -1.0479722023010254, -1.028617024421692, -0.6667669415473938, 1.5735583305358887, 3.0393693447113037, -0.6177296042442322, 1.2748420238494873, 0.9509825706481934, 1.1238930225372314, -1.6795226335525513, -0.8843985795974731, -1.3239692449569702, -0.9252637028694153, 0.25967514514923096, -0.30544939637184143, 0.7952330708503723, 1.1885638236999512, -0.5548747181892395, 0.7849673628807068, -0.40627002716064453, -1.1218198537826538, -0.9665388464927673, 0.1900707483291626, 0.12196092307567596, 0.9338348507881165, 0.21744100749492645, 1.0232945680618286, 0.6629250049591064, 1.7294039726257324, -1.726615309715271, 1.9752436876296997, 0.3440863788127899, 0.8047047257423401, 1.1394742727279663, 0.31302475929260254, 0.9580549001693726, -0.025270599871873856, 1.656893253326416, 0.9901732206344604, 0.5753155946731567, 0.47247859835624695, 0.30856722593307495, -0.2014470100402832, -0.33723050355911255, 0.07176852971315384, 2.2178783416748047, -0.7431222796440125, -0.41673940420150757, 0.780277669429779, 0.6191601753234863, -0.41383904218673706, 0.6792174577713013, 0.325514018535614, 1.1016563177108765, 0.4682272970676422, -0.594713032245636, 0.5094255208969116, 1.6159132719039917, 0.1257748007774353, 1.798674464225769], [0.8971693515777588, 0.05661611631512642, -1.0106502771377563, -1.6087141036987305, -0.6115766167640686, -0.7134410738945007, 0.0027278049383312464, -0.17821386456489563, -1.070915699005127, 0.968258261680603, 0.8789452314376831, 0.1459786295890808, 1.0485526323318481, 1.3290956020355225, 0.42199280858039856, -0.23158544301986694, -0.12547500431537628, 0.44469892978668213, 1.7504814863204956, -0.42924928665161133, -0.667051374912262, 0.9642215371131897, -0.7833637595176697, 1.1661344766616821, -0.3784746527671814, -1.6237895488739014, 0.8539272546768188, -2.1314265727996826, 0.6872564554214478, -1.4210511445999146, -0.5032159686088562, -1.2316021919250488, 1.6981451511383057, 0.9505279064178467, -0.07872425764799118, -0.6235530376434326, -1.1984459161758423, 1.6416938304901123, -0.8702976107597351, -0.8242732882499695, 2.0956313610076904, 0.113432876765728, -0.5091778039932251, -0.4349082410335541, 1.048200249671936, -1.27460515499115, 0.47265344858169556, -0.4134630560874939, -0.39606088399887085, 0.5619295239448547, -0.5535001754760742, -0.8036521673202515, -0.3038230240345001, -0.13386216759681702, -1.5015068054199219, 0.04232054203748703, -1.3920913934707642, -0.8453631401062012, 0.07140551507472992, -0.17761924862861633, 1.484734296798706, 0.37823814153671265, -0.6016484498977661, 0.998950183391571, -0.8798553347587585, -0.3326059579849243, 0.05289169400930405, -0.2316976636648178, -0.7502137422561646, -0.183394655585289, 0.24510644376277924, 0.8771106004714966, 0.7173396348953247, -0.8794572949409485, 1.9552547931671143, -1.3911556005477905, 1.8191378116607666, 0.2523258626461029, -0.7233962416648865, 0.12742894887924194, 1.5033254623413086, -1.0653880834579468, 0.06657082587480545, -0.00479502510279417, -0.5830090045928955, -0.7274224758148193, -1.4696879386901855, -0.7775247097015381, -0.6050830483436584, 0.6722707152366638, 0.34474611282348633, -1.3170478343963623, -0.445639431476593, -0.3179851174354553, -0.035640180110931396, 0.3738097846508026, -0.07725932449102402, 0.33756953477859497, 1.2536170482635498, -0.5792704820632935, 0.1317221075296402, -0.5410860180854797, 1.9534645080566406, -0.8170502185821533, -1.0268748998641968, -0.9942634701728821, -0.8630668520927429, -0.2646985352039337, 0.025094982236623764, 0.5562182068824768, -0.5187197923660278, 1.018265724182129, 1.6217670440673828, 0.935670018196106, 1.1502822637557983, 0.8972631096839905, -1.696549415588379, -2.4767329692840576, 0.27440449595451355, -0.4557151198387146, -0.33989766240119934, -1.1631306409835815, -0.33539754152297974, -0.3917333781719208, -0.9058721661567688, -0.07091650366783142, -0.4023388922214508, 0.7023298144340515, 1.2554081678390503, -0.5893977284431458, -0.062071360647678375, -0.057873986661434174, -0.5403233766555786, 0.3596947491168976, -1.1246700286865234, 2.7761292457580566, 0.727799654006958, 0.5303611159324646, -0.1826157122850418, 0.3847184479236603, -1.1915220022201538, 0.1539095789194107, -2.227501392364502, 0.6938080787658691, -0.8417882323265076, -1.9046921730041504, -1.1441251039505005, -2.2684407234191895, -0.5673530697822571, 0.6717218160629272, 0.19529440999031067, -1.0408272743225098, 1.0749090909957886, -1.705726146697998, -0.9908837676048279, -1.5815656185150146, -0.06459049880504608, 0.8718958497047424, 0.6555451154708862, 1.6748244762420654, -0.733193576335907, 0.16859059035778046, -0.3788049817085266, -0.7493236660957336, 0.15209102630615234, 0.6117474436759949, -0.6405001282691956, -0.46302202343940735, 0.919032633304596, 0.5516721606254578, -0.3078295886516571, -1.2575461864471436, 0.006890064105391502, 0.38004210591316223, -0.9458718299865723, -1.7280690670013428, 0.6467838883399963, -0.5095735192298889, -0.678881049156189, -0.3953222334384918, -0.5989083051681519, 0.4743652641773224, -0.03391743451356888, -0.28613051772117615, -0.8506006598472595, 3.3713324069976807, 1.330353856086731, 1.0773364305496216, -1.3926036357879639, 1.2931928634643555, -0.08643484115600586, -0.3574264347553253, -0.26540055871009827, -0.35071831941604614, -0.6730426549911499, -0.2511580288410187, -0.26617148518562317, 0.8803545236587524, -1.6611100435256958, -0.34860751032829285, -1.3620723485946655, -0.3132130205631256, -0.17654526233673096, -0.5388215780258179, -0.8437690734863281, -1.0738635063171387, -0.6819486618041992, -0.4140091836452484, 1.3380502462387085, 0.4506245255470276, 0.6973522901535034, -0.4806813597679138, 0.21128994226455688, -0.3000243008136749, 0.6508235335350037, 0.040612854063510895, -0.10226307064294815, 1.2566802501678467, -0.6874657869338989, -1.0369665622711182, 0.22188830375671387, 2.0994293689727783, 0.7281211018562317, -0.31032994389533997, 0.6910980343818665, -0.06532996892929077, 0.7544072270393372, -0.025399817153811455, 0.7932397723197937, -0.6404440999031067, 0.1475631445646286, -0.03131701424717903, 1.9641984701156616, -0.8902722001075745, 0.13826917111873627, 0.3870108425617218, 0.09571844339370728, -1.5807435512542725, -0.03749720752239227, -0.40945252776145935, 0.8752302527427673, 0.00032554223434999585, 0.5604655742645264, 1.3454169034957886, 0.16731275618076324, 1.0263707637786865, 1.5861637592315674, -1.1880313158035278, 0.4968251585960388, 1.368436574935913, 0.6077694892883301, 1.7033231258392334, 0.23429884016513824, 0.5290988087654114, 0.8136811852455139, 1.0169148445129395, 0.14731141924858093, -0.1082930639386177, -1.1580111980438232, 0.25793927907943726, 1.2874090671539307, -0.4971473217010498, 0.47058752179145813, -2.232893705368042, -0.2868883013725281, 1.303322672843933, -0.7759732604026794, -0.40655210614204407, 0.5022561550140381, -0.2225039005279541, 1.2326854467391968, -0.9841914176940918, 0.25720489025115967, 1.0096489191055298, 0.9506114721298218, 0.2063933163881302, -1.0333466529846191, 0.143238365650177, -0.6029765605926514, -0.47820013761520386, 0.6391278505325317, 0.3656146228313446, -0.386404812335968, 1.2556803226470947, 1.2750924825668335, -0.7238328456878662, -1.231602430343628, 1.9034546613693237, -0.6120275855064392, 0.7992020845413208, 1.0641016960144043, -0.30350372195243835, 0.45432600378990173, 0.2181757539510727, 0.8615519404411316, 0.7274911403656006, -0.9512549042701721, 0.556020975112915, -1.3807939291000366, 0.2559310495853424, 0.011823671869933605, 0.7111716866493225, 1.2627633810043335, 0.6000339388847351, -1.6228218078613281, -1.618282437324524, 0.766278862953186, 1.3042657375335693, 0.18295937776565552, -0.6001356840133667, 0.14369475841522217, -0.032226987183094025, 0.5842766761779785, -0.9537155032157898, -0.5673704147338867, -0.5334585309028625, -0.10425673425197601, -0.5922744274139404, 1.4133847951889038, -0.4022313356399536, -0.1313813030719757, -0.3994855284690857, 0.780890941619873, -0.6654171347618103, -1.0176810026168823, 0.08608820289373398, 0.07455092668533325, -0.549920380115509, -0.10055329650640488, 0.05660604313015938, -0.2862512767314911, 1.0576872825622559, 0.582760751247406, 0.07787362486124039, -0.030282622203230858, 0.3920024335384369, -1.1966283321380615, 2.5518558025360107, 0.031118100509047508, 0.33735784888267517, 1.767435908317566, 0.10852556675672531, -1.9081603288650513, -0.26858681440353394, -0.11170988529920578, 0.01989704743027687, -0.36711007356643677, 0.06605534255504608, 1.451244831085205, -1.6168626546859741, -0.4789828956127167, -0.6921939253807068, -0.972936749458313, -0.6188876628875732, 1.604500412940979, 1.7541815042495728, 1.1286203861236572, 2.0966873168945312, 0.489665687084198, -0.8378189206123352, 0.3669951558113098, -1.098093032836914, 1.2122312784194946, 0.9264668226242065, -0.7797755599021912, 0.7969911098480225, -0.05797054246068001, 1.0110024213790894, 0.3835202753543854, 1.181126594543457, -0.04595975577831268, 0.2828120291233063, 0.47088688611984253, -0.19191227853298187, 2.103395700454712, 0.9138035774230957, -0.378150999546051, -1.8933253288269043, -1.24077570438385, -1.6267305612564087, -1.2848353385925293, -1.4491759538650513, 0.8911596536636353, 0.28918221592903137, 0.20504635572433472, -0.2091156393289566, 0.7372682690620422, 1.053534746170044, -0.29809170961380005, -0.6276116371154785, -0.14779438078403473, 2.299636125564575, 0.9795930981636047, 0.14773918688297272, 0.0035943614784628153, -2.4378674030303955, 0.5113235116004944, 0.09672242403030396, 0.5858057141304016, 0.7814090847969055, 0.31339216232299805, 0.8986876010894775, 0.8773834109306335, 0.2967771291732788, -1.9722148180007935, 1.5483319759368896, -0.4694375991821289, 0.8229405283927917, 0.6838477849960327, 0.48554983735084534, -0.5299975872039795, -0.628912091255188, 0.705094575881958, 0.22312681376934052, -1.566064476966858, -1.6832650899887085, -0.12672364711761475, -0.7308078408241272, -1.1963931322097778, -0.08526793867349625, 2.0782277584075928, 0.32826104760169983, -1.5797252655029297, -1.2681490182876587, 0.04764176532626152, -1.2936574220657349, 0.4366549253463745, -0.8107320070266724, -0.5659474730491638, -0.356603741645813, 0.7813522219657898, -0.26883089542388916, -2.381286144256592, 0.9769611358642578, -1.1174849271774292, 0.11544926464557648, -1.0854800939559937, 0.8218304514884949, -1.7852153778076172, 0.6141209602355957, -0.3353058099746704, -0.004554332233965397, 0.9598084092140198, 0.3103700578212738, 0.5486604571342468, 0.8188827633857727, -0.38298478722572327, 1.545682668685913, -0.058644600212574005, 0.03212279453873634, 0.6583557724952698, 0.4266285002231598, -0.2524062693119049, 1.5371328592300415, -0.43784695863723755, 0.07523459941148758, -2.322606325149536, -0.544559895992279, 1.2315338850021362, 0.29492467641830444, 0.33148398995399475, 0.5111571550369263, -0.1506284475326538, -1.4779859781265259, -1.270609736442566, 0.663038432598114, 0.12709547579288483, -0.24835212528705597, 1.594020962715149, -1.4355305433273315, 0.2400660514831543, 0.2071094959974289, -0.7284725904464722, 0.3158835172653198, 2.290982723236084, -0.36189812421798706, 0.7980022430419922, -0.060884613543748856, -0.8173529505729675, -0.7238376140594482, -1.3856202363967896, -0.4942420721054077, 0.5506780743598938, 0.01523127406835556, -0.8707906007766724, 0.6950182914733887, -0.20294831693172455, 0.2736087441444397, -0.5140708684921265, -0.13184231519699097, 2.738919734954834, 1.0909160375595093, -0.21469946205615997, -0.9093971252441406, 0.49083077907562256, -1.1310451030731201, -0.4283161163330078, 1.8987895250320435, -0.1063251718878746, -0.7993633151054382, 0.14043323695659637, 0.6427253484725952, 0.2666163444519043, -0.15768200159072876, 0.36081522703170776, 0.16517816483974457, 0.36191821098327637, 0.5947113633155823, 0.22347593307495117, 0.9660264253616333, -0.41586577892303467, -0.7643123865127563, -0.38223350048065186, 0.4195084869861603, 2.158418655395508, -1.2357174158096313, 0.5280047059059143, -0.27727803587913513, 0.0023227680940181017, 0.370659738779068, 0.5118589401245117, 0.4408813416957855, 0.41496577858924866, -0.9381692409515381, -1.0670850276947021, -0.9269167184829712, -0.4807840585708618, 2.080862283706665, 0.6013576984405518, -1.1496503353118896, 1.234033226966858, -0.9949101805686951, -0.9836141467094421, 1.004342794418335, 0.004838613793253899, -0.18023040890693665, -1.0123701095581055, -0.8418072462081909, -0.09398557990789413, -0.07648014277219772, -1.1997692584991455, -1.1610304117202759, -1.119850754737854, -0.193617582321167, -0.1519819051027298, -1.615441083908081, 1.5651990175247192, 0.42580607533454895, -1.0678201913833618, 0.06493556499481201, -0.9341083765029907, 1.5436865091323853, -0.7827311158180237, 1.239274263381958, 0.7389093041419983, 0.21043573319911957, -0.15003401041030884, -0.18236039578914642, 0.021745214238762856, 2.0599911212921143, -0.4369039833545685, 0.40742871165275574, 1.294127345085144, -1.0141135454177856, 1.0420199632644653, 0.45027029514312744, 1.0283799171447754, 1.4743939638137817, 0.39065465331077576, -0.22593313455581665, -0.41476526856422424, 1.0935951471328735, -0.22074952721595764, -1.2334084510803223, -0.07992154359817505, 0.3638143241405487, 0.6471362709999084, -0.3249078392982483, -0.4650031626224518, 0.02198519930243492, -0.15996554493904114, 0.9796141982078552, 0.6067620515823364, 0.4760110378265381, 0.3050033450126648, 0.6688528060913086, 0.2645857334136963, 0.4760100245475769, -0.5034617185592651, -1.270475149154663, -0.248448446393013, 0.7943463325500488, -1.2998279333114624, 0.630439281463623, 0.011665304191410542, 0.5006110072135925, -1.3956800699234009, 1.0259857177734375, -0.1311630755662918, 1.6031904220581055, 0.7163376212120056, -0.7136551141738892, 0.6568268537521362, -0.3270304501056671, 0.7174392342567444, -0.8173498511314392, 0.9119016528129578, 1.5840363502502441, -0.8013803362846375, -0.23631486296653748, 0.99267578125, 0.4668511748313904, 0.08453142642974854, -0.6287659406661987, 0.21184107661247253, -0.49237746000289917, -0.8104163408279419, -0.14115050435066223, 0.8328348994255066, 2.1890649795532227, -2.370121955871582, -0.8506137132644653, 1.199296474456787, 0.9514006972312927, 0.057498231530189514, -0.34801605343818665, -0.7567342519760132, 1.1615620851516724, 1.6060163974761963, 1.2571377754211426, -0.5513947606086731, -0.4467732012271881, 0.05210578441619873, -1.9020271301269531, 0.790901780128479, -0.42249608039855957, -1.3166321516036987, 1.05925452709198, -0.813147246837616, -1.1983842849731445, -1.0574651956558228, -0.9280616641044617, 1.4194663763046265, 0.3570117652416229, -0.7931563258171082, 0.9331104159355164, 1.1766490936279297, 0.696986973285675, -0.2607761323451996, 1.0120357275009155, -1.1817903518676758, -0.7128759026527405, -0.06933102011680603, 1.265518307685852, -1.7123020887374878, 0.4901511073112488, -0.3360409438610077, 1.7795406579971313, -2.329866409301758, -0.3523067533969879, -0.7215735912322998, 2.0278122425079346, -0.058497630059719086, -1.5106866359710693, 0.028244227170944214, -0.8012270927429199, 2.1013214588165283, -1.8534578084945679, -0.3582381308078766, -0.6466836929321289, -0.22015972435474396, 0.524039626121521, 0.5631911158561707, 0.4771106541156769, 2.1243417263031006, -0.07762191444635391, 2.105541944503784, 1.132534146308899, -0.23789086937904358, 0.2603573799133301, 0.6554877161979675, 1.3445740938186646, 1.4890613555908203, 0.044350139796733856, -1.3401381969451904, 1.644169807434082, -0.8950279951095581, -0.7887240648269653, -0.9188382625579834, 0.6740971207618713, 0.846052885055542, -0.2230881303548813, -2.2805514335632324, 0.936206042766571, -0.4360705018043518, -1.3549256324768066, 2.336338996887207, -0.20083624124526978, 2.248879909515381, 0.7649094462394714, -0.49910175800323486, -0.29044365882873535, -1.3095827102661133, -0.23137150704860687, -0.31504252552986145, -0.5783875584602356, 0.3300791382789612, -0.17000702023506165, -1.3787540197372437, 0.06192341446876526, -0.752538800239563, -0.8271790742874146, 1.040047287940979, 0.7340003848075867, 0.8769251704216003, -1.4632574319839478, 0.9667391777038574, -0.4837186336517334, 1.5212420225143433, -0.18391163647174835, 0.7804041504859924, -0.5211259722709656, 0.16532938182353973, -0.6774954795837402, 0.775443971157074, -0.44318991899490356, -0.9180917143821716, 1.0175038576126099, 0.11809918284416199, 0.9857202172279358, 1.577997088432312, -1.0407979488372803, -0.5206755995750427, -2.1086015701293945, 0.6726887226104736, 0.8753436803817749, -0.26635539531707764, -0.6382167339324951, 2.1165871620178223, -0.6386306881904602, 0.10307769477367401, 0.7523936033248901, 0.4499583840370178, -0.9064958095550537, -0.5299257040023804, -0.4636124074459076, -1.4521161317825317, 0.7292398810386658, -0.7209271192550659, 0.08660084009170532, 0.2125677466392517, -0.7374677658081055, 0.3688402473926544, 0.11947677284479141, 0.9657782316207886, -1.7011696100234985, 0.5667636394500732, -0.24127963185310364, -0.8679138422012329, -0.0013482996728271246, -0.04937439411878586, -0.5111403465270996, 0.7718349695205688, -0.09677159786224365, -0.16461074352264404, 0.05700032413005829, 1.4285820722579956, -1.6293658018112183, 0.5003174543380737, 0.4470446705818176, 0.19626571238040924, -0.15637651085853577, 0.815724790096283, 0.19987989962100983, 2.2961678504943848, 0.7815049290657043, -0.6935387849807739, 1.6110670566558838, 0.3499065637588501, -0.32636991143226624, -0.05910836532711983, 0.030216876417398453, -0.7943510413169861, 2.339069366455078, 1.4159047603607178, 1.1868550777435303, 0.2614189684391022, -0.5644692182540894, -1.2849129438400269, -1.4481648206710815, -0.06567603349685669, 0.615237832069397, 0.6483445763587952, -0.3192896842956543, -0.3198525905609131, -0.11597971618175507, -0.7515349388122559, 0.2886941432952881, 0.47312992811203003, 0.08765539526939392, 0.13159847259521484, -0.5460637211799622, -1.0653636455535889, 0.5603070855140686, -1.802720308303833, -0.90267413854599, 1.2445122003555298, -0.03753950446844101, 1.1422960758209229, -1.0490703582763672, -0.23142704367637634, 0.038961298763751984, -1.3399955034255981, 1.7798337936401367, -0.12397682666778564, 1.363335371017456, -1.08379328250885, -1.5568255186080933, 0.9699430465698242, 1.0169545412063599, -1.0165257453918457, -0.5108169317245483, -1.2507331371307373, 0.43054184317588806, 0.8287178874015808, -0.1478729099035263, 1.3139699697494507, -0.748240053653717, -1.617084264755249, -0.49591708183288574, 0.33410021662712097, -0.04801288992166519, -0.25846633315086365, -0.3593464195728302, 1.1844438314437866, 0.5639849305152893, 0.5487126708030701, 2.42867374420166, 0.4590558409690857, -0.09440213441848755, 0.2643181383609772, -1.0921111106872559, 0.6130790710449219, -0.518665611743927, 1.3495413064956665, 0.43284446001052856, 0.8258367776870728, -0.18754372000694275, 0.32712438702583313, -0.7376652359962463, -1.662345051765442, 0.049224622547626495, -1.0920400619506836, -0.6239332556724548, 0.7053657174110413, 0.3517199158668518, -1.2050060033798218, -0.7044920325279236, -0.5029600262641907, -1.8113336563110352, -1.3739104270935059, 1.1084765195846558, 1.2264372110366821, 1.184497356414795, 0.5628332495689392, -0.6547898054122925, 0.11147862672805786, 0.9024035334587097, 0.03318263590335846, -2.068882703781128, 0.826426088809967, -0.8465994000434875, 1.0065383911132812, 0.6115554571151733, -0.597414493560791, -1.2390974760055542, 1.5577088594436646, 1.5344312191009521, 0.573282778263092, -0.3682510554790497, -0.8545337319374084, -0.4279566705226898, -0.1223759725689888, 0.9871492385864258, 0.21585699915885925, 0.23139896988868713, -1.9357393980026245, 2.631547212600708, 1.0083781480789185, 1.1722663640975952, 0.050795480608940125, 0.8647649884223938, 1.7817158699035645, 1.0206698179244995, 0.9174293279647827, 0.9861916899681091, -0.22673681378364563, 1.9781408309936523, -0.04666934907436371, -2.096686363220215, -0.26397982239723206, 0.089804507791996, 0.9675428867340088, 0.18632490932941437, 0.07131623476743698, -0.6165796518325806, 1.7128320932388306, 0.23552480340003967, 0.12891925871372223, 1.297635555267334, -1.742092251777649, -0.967366635799408, -1.6153486967086792, 0.9646849036216736, 0.896384596824646, -0.12634585797786713, 0.15310895442962646, -0.19514600932598114, 1.5839531421661377, 0.025240525603294373, -0.07815080881118774, 0.5105558037757874, 0.6651210188865662, 0.2270185947418213, -0.7740663290023804, -0.6232223510742188, -0.19625815749168396, -0.19326171278953552, -0.6791056990623474, 0.3140642046928406, 0.26223811507225037, -1.1803933382034302, 1.2028570175170898, -0.5376110076904297, 1.7969478368759155, -0.22136203944683075, -1.1027607917785645, -1.1745504140853882, 1.759029746055603, -0.027474042028188705, -1.0480360984802246, 0.27089741826057434, 0.35536929965019226, 0.5440462231636047, 0.2550515830516815, 0.3162652552127838, -0.27046385407447815, -0.5114492177963257, 0.19333045184612274, -0.09888520836830139, 0.10596773773431778, -1.783638596534729, 0.2093471735715866, 1.8812803030014038, -2.483537197113037, 0.061391375958919525, 0.7315937280654907, 0.031435996294021606, -1.1522523164749146, 0.6716508269309998, 0.6323802471160889, 0.7249767184257507, 0.3043132424354553, -0.2068580538034439, -1.7726140022277832, 0.02937406674027443, -2.284785032272339, -0.19501042366027832, -1.0437703132629395, 1.4310053586959839, -0.5843514204025269, 0.10491122305393219, -0.3782133460044861, 0.7866902947425842, 0.12518388032913208, -1.0011399984359741, -0.14981791377067566, 0.5008299946784973, -0.42571407556533813, -0.8404018878936768, -0.16023549437522888, -0.3955579400062561, -1.1252551078796387, 0.7651901245117188, -0.7817156314849854, 1.0038467645645142, 1.854962706565857, -0.32107722759246826, 0.3476182520389557, 0.34663039445877075, 1.3430075645446777, 0.10300799459218979, 0.08624549955129623, 0.37121695280075073, 2.2732994556427, 2.0668554306030273, -0.3199981153011322, 0.672051191329956, 0.4752400815486908, -0.3992573022842407, 1.0593199729919434, -0.7191783785820007, 0.5667416453361511, 0.5289117693901062, 0.5145114064216614, -1.0215076208114624, -0.5040215849876404, -1.416314959526062, 0.5588850378990173, 0.13831454515457153, 0.5893706679344177, -1.2774969339370728, 0.17682944238185883, -1.306246280670166, -0.5220231413841248, 0.49487176537513733, -0.44653257727622986, -0.226880744099617, 1.626649022102356, 1.086826205253601, 2.41481876373291, 0.5545206069946289, 1.0498785972595215, 0.12037816643714905], [-0.5809836387634277, 0.13074065744876862, -1.7055281400680542, 0.5493226051330566, 0.2429426908493042, -0.757318913936615, -1.5791847705841064, 0.7521079182624817, -1.0481983423233032, -0.7455534934997559, 0.9393237829208374, 0.5139384865760803, 0.47026702761650085, -0.12932232022285461, 1.494834542274475, -0.2056790292263031, 0.708275318145752, -0.8128069043159485, -0.1528371274471283, -0.0035916210617870092, -0.9923158288002014, 0.8608376383781433, -0.9167112112045288, -0.8282567262649536, -0.41046297550201416, 1.81429922580719, 1.1783239841461182, 2.685817003250122, 0.5922221541404724, -0.5930907726287842, -0.1607176661491394, -1.0978267192840576, 0.2941371500492096, -2.48034405708313, -0.4080674350261688, 1.8733556270599365, 0.40386855602264404, 3.918759346008301, -0.531256377696991, 0.8193348050117493, 0.3162537217140198, -0.2495701014995575, -0.40859681367874146, -0.08095806837081909, -1.8415331840515137, -1.582413673400879, -0.9151487350463867, -0.5969197750091553, -0.7684901356697083, 0.24383853375911713, 0.07391037046909332, 0.8166548013687134, 0.4938853681087494, -0.09071444720029831, -2.1360840797424316, -0.7938923239707947, -2.044572353363037, -0.5670431852340698, -1.1849830150604248, -1.3189036846160889, -0.0960899293422699, -0.4158118963241577, 1.3172067403793335, 0.6119872331619263, -0.9249259233474731, -0.32548627257347107, -0.6659186482429504, -1.2398616075515747, -0.1492777168750763, -1.7334578037261963, -1.7828978300094604, 1.4338560104370117, 0.6775330901145935, -1.2533918619155884, -0.03225279599428177, 0.2675734758377075, 1.8380481004714966, 0.08699516206979752, 0.019026953727006912, -0.8929303288459778, -0.9382907152175903, 0.23554301261901855, -0.1907760202884674, 0.22022037208080292, -1.033929467201233, 0.5521718859672546, -0.2445395290851593, -0.8678028583526611, -1.3815038204193115, -0.6088190674781799, 0.7629148960113525, 0.9054195284843445, -0.5736029744148254, -1.1572213172912598, 1.0000675916671753, -0.8620806336402893, -0.3032500147819519, 1.2889498472213745, -0.5557371377944946, 0.9337327480316162, 0.16168522834777832, -0.23890414834022522, 0.9369502663612366, 0.9344736933708191, 0.46747955679893494, 0.44655951857566833, 1.6601073741912842, -1.0777692794799805, -1.0955959558486938, -0.5053087472915649, -0.057528600096702576, -0.5482121109962463, -0.2311045229434967, -0.6755318641662598, 1.14274001121521, 0.5020014643669128, -0.15966635942459106, 0.48131516575813293, -0.659115195274353, 0.049929071217775345, -0.6748784184455872, -1.7392100095748901, 0.4636937975883484, -0.1688840389251709, 0.011467257514595985, 1.6170475482940674, -0.18951193988323212, 1.2302939891815186, -0.7977975010871887, -1.749825358390808, -0.27976807951927185, -1.070045828819275, -0.4879545271396637, -0.007221181411296129, -0.5181483626365662, 2.0818212032318115, -0.3226093649864197, 0.9052668213844299, -0.4710163474082947, -0.8676308393478394, 0.2951291501522064, -1.443841814994812, -0.5514160990715027, -0.24605169892311096, -0.14829899370670319, 0.8814186453819275, 0.8118615746498108, 1.5171394348144531, 0.2973119914531708, 0.2127763330936432, 0.33786603808403015, 0.19086512923240662, -0.6789726614952087, -1.0637011528015137, 1.0474129915237427, -1.104087233543396, -1.047357201576233, -2.5147271156311035, -0.31424757838249207, 0.6929028034210205, 0.6078414916992188, 0.5866135954856873, -0.9754668474197388, -1.2673628330230713, 1.141320824623108, -1.1096714735031128, -0.16844157874584198, 1.2790533304214478, -1.1478159427642822, -1.4598618745803833, 0.3541802167892456, 0.7582669854164124, 0.18329699337482452, -1.4232293367385864, 0.11896054446697235, 1.1161818504333496, 0.8000902533531189, -0.5610337853431702, -0.31814366579055786, -0.21443846821784973, -0.27871519327163696, 0.046376217156648636, -0.45419952273368835, 1.202377200126648, 0.05869891494512558, -0.021427327767014503, -0.7028467655181885, 0.4087400734424591, -0.23819494247436523, 0.8139927983283997, -0.2626490592956543, 0.46948862075805664, 0.018889112398028374, -0.33312666416168213, -0.22959384322166443, -0.5661532878875732, -1.6005651950836182, -1.530366063117981, -0.8381344676017761, -0.5973080992698669, -0.5298821330070496, -0.17034587264060974, -0.3248843848705292, 0.6590963006019592, -0.8293572664260864, -0.1988057643175125, -0.4694332778453827, 0.015031312592327595, -0.8834500908851624, 0.36237433552742004, 0.734632670879364, -1.61484694480896, 1.367289423942566, 1.2723110914230347, -0.9431845545768738, 0.18178248405456543, -1.0781753063201904, -0.15249522030353546, -0.5766489505767822, -0.5154927372932434, -0.7722625136375427, 0.6848835349082947, 0.14186660945415497, 1.5237733125686646, -1.0261204242706299, 0.08290520310401917, -0.28731584548950195, 0.970102071762085, -1.6298363208770752, -0.39232325553894043, 0.4587065577507019, 0.22200259566307068, -0.6406239867210388, 1.0192455053329468, -0.8675622344017029, 0.38465824723243713, 0.29445865750312805, 2.34356427192688, -0.25642824172973633, 0.8532869219779968, -0.5068650841712952, 0.8971970081329346, 1.4970802068710327, -0.46747833490371704, 1.9040008783340454, -0.4898024797439575, -0.3695715665817261, -1.4905502796173096, -1.1418780088424683, -0.9453094601631165, -0.08922974765300751, 2.093385934829712, 0.2491145133972168, 0.30229809880256653, 0.31501293182373047, 0.22751709818840027, 0.8924173712730408, -0.9572679996490479, 1.2092643976211548, -0.1828444004058838, 1.6742212772369385, -0.40041694045066833, 0.7352133989334106, -2.7635209560394287, -0.06657882779836655, 0.3863663971424103, -0.3365981876850128, 1.2256742715835571, 0.6015768051147461, 0.28979143500328064, 1.982435703277588, 0.4604125916957855, -0.8585980534553528, 2.003596305847168, 1.6979855298995972, 0.5285438895225525, 0.976056694984436, -0.3106065094470978, 0.2741355299949646, -1.1076865196228027, -0.829457700252533, -0.19887566566467285, 0.2755691409111023, 0.22170603275299072, 0.6213566660881042, -0.8109187483787537, -0.4685191512107849, 2.0154144763946533, -0.9532480239868164, -1.2060953378677368, -0.0027839073445647955, 0.054174624383449554, 1.194916009902954, -1.163395643234253, -1.6531373262405396, -0.3507586717605591, -1.1053494215011597, 0.6430436372756958, -0.18578721582889557, -0.44334644079208374, -0.3997223973274231, 0.035093843936920166, -0.44957637786865234, -0.551106333732605, 0.6679208278656006, 0.5341891646385193, -1.0896167755126953, 0.3999083936214447, 1.6677531003952026, -0.049894385039806366, -1.8432486057281494, -0.03863426297903061, -0.5734081268310547, 0.47434237599372864, -0.15058894455432892, -0.03060712292790413, 2.399871826171875, 0.5535845160484314, 1.348897099494934, 0.8812601566314697, -1.6793186664581299, -1.9102795124053955, -0.26157346367836, 0.3434949517250061, -0.8290276527404785, -0.16981010138988495, 0.15425902605056763, -0.1464444249868393, 0.735159158706665, -0.6830900311470032, -1.1037614345550537, 1.9731992483139038, -1.235892653465271, 0.3375250995159149, -1.4578579664230347, -1.0614787340164185, -0.1680656522512436, 0.759124755859375, -1.1610060930252075, 1.015891432762146, 0.5606192946434021, 1.928576111793518, 0.2850147485733032, 0.03219757601618767, 0.96198570728302, -1.6584655046463013, 0.3149976432323456, -0.19145123660564423, -0.08169467747211456, 0.08473216742277145, -0.21458372473716736, 0.24188925325870514, 0.9954432249069214, 3.2649433612823486, -0.36453452706336975, 0.018281864002346992, 0.9562209248542786, 0.5823752284049988, 0.5933102369308472, -2.1337382793426514, -1.4457000494003296, -0.5223597288131714, -1.611969232559204, 0.0483415424823761, -0.5135368704795837, -1.036839246749878, 1.2511032819747925, 0.16032938659191132, -1.8343502283096313, 2.6765663623809814, 0.20599350333213806, 2.1755385398864746, -0.5875557661056519, 1.5847060680389404, 1.0284984111785889, 1.672330617904663, 2.336397886276245, 1.6909388303756714, 0.29634276032447815, -1.6748452186584473, -1.1093716621398926, -0.9193552732467651, 1.8928098678588867, 0.2166881561279297, -0.3984215259552002, -0.4497523605823517, 1.3625848293304443, 0.41232728958129883, 0.7815162539482117, 0.6966974139213562, 0.24451109766960144, 1.6131304502487183, -0.6033729910850525, -0.42745816707611084, -0.8013322353363037, -1.1124277114868164, -0.465667188167572, -1.2004789113998413, 1.7106437683105469, -0.7547706365585327, 0.5813996195793152, -0.30372411012649536, -0.8411908745765686, 1.0613059997558594, -1.5839372873306274, -0.16010956466197968, 0.21835249662399292, -0.11050494015216827, 1.9281927347183228, -0.0417521707713604, 0.5900123715400696, 1.2446081638336182, 0.7939267158508301, 2.87203049659729, -1.1682703495025635, -0.9330070614814758, -0.7565548419952393, 0.6259394884109497, 0.003501899540424347, 0.18501532077789307, 0.37393051385879517, 0.7902150750160217, 0.41175326704978943, 0.5443770289421082, -0.13392066955566406, 0.5279476642608643, 1.0781793594360352, 0.43428999185562134, 0.8296203017234802, 0.17925716936588287, 0.6243030428886414, -0.18381652235984802, 1.5007257461547852, 1.6851524114608765, 1.9639524221420288, 1.1240118741989136, 0.9719865322113037, 1.0280606746673584, 0.2391681969165802, -0.8036704063415527, 0.9644759893417358, -0.8726935386657715, 0.5071410536766052, 1.0366041660308838, -0.11352674663066864, 0.8940572142601013, 0.024300310760736465, 0.6584630012512207, 0.03569142892956734, 0.647067129611969, -0.6542880535125732, -1.5152262449264526, -1.4403256177902222, -0.2726321518421173, 0.4636044502258301, -0.10328374803066254, 0.21086154878139496, 1.7246930599212646, -0.7566908597946167, -1.0221705436706543, -1.2875967025756836, 0.7477281093597412, 1.3295023441314697, -0.5393884778022766, 2.463573694229126, -0.14074209332466125, 1.0732146501541138, 0.7230295538902283, -1.5525115728378296, -0.07357947528362274, -0.09174880385398865, -0.00733983376994729, 1.45978844165802, 0.7067390084266663, 0.12257780879735947, -0.5464800000190735, -0.1931755542755127, -1.0908286571502686, 1.9713108539581299, -0.4168623685836792, -0.37424543499946594, -0.6475672125816345, 1.6168606281280518, -0.20441552996635437, -0.5870880484580994, -0.04845794290304184, -0.13508446514606476, 0.553372323513031, 1.6505711078643799, 1.0732313394546509, -0.9127355813980103, 0.7547409534454346, -0.37738528847694397, 0.15383198857307434, -0.7218076586723328, -0.9314165115356445, 0.5435042381286621, 0.7603700757026672, 0.579155683517456, -1.482830286026001, -0.9792305827140808, -0.22242368757724762, -0.3740503787994385, 1.7273601293563843, -1.9016681909561157, -0.12429111450910568, 0.5529503226280212, -0.5304631590843201, -0.4497774541378021, -1.136462926864624, 0.8816651701927185, 2.440403699874878, 1.04924738407135, -0.6425849199295044, -0.3783169984817505, -1.4559378623962402, -1.6032792329788208, 0.5255128741264343, 0.6189477443695068, 0.226331889629364, -0.18610548973083496, -0.42505693435668945, 0.7657203078269958, -0.18900784850120544, -0.41647040843963623, -0.2621733844280243, -0.7628698945045471, -1.1844784021377563, 0.4022032916545868, -0.5644121170043945, 1.6252555847167969, 0.4726484715938568, -0.05804125592112541, 1.6340864896774292, -1.283371090888977, 0.2912245988845825, -0.21276679635047913, 1.2319501638412476, 0.7496370077133179, 0.9104843735694885, 0.14858688414096832, -0.04748304560780525, -1.2193326950073242, -1.3053302764892578, 0.42236021161079407, 0.5713563561439514, -0.36053985357284546, -0.8128560185432434, 1.111862063407898, 0.4416532814502716, -0.19786961376667023, 1.044442057609558, -0.19915181398391724, 1.4406535625457764, 0.2959776222705841, -0.06484900414943695, -0.3387242257595062, 0.3959410786628723, -0.08763343095779419, -1.6439619064331055, -0.017633013427257538, -0.6957953572273254, -0.9575057029724121, 0.5380262732505798, -0.8743293881416321, 0.9821251034736633, 0.08060900866985321, -0.13724182546138763, 0.26796093583106995, -1.0718728303909302, -0.5709264278411865, 0.443833589553833, 0.8761969208717346, -0.7256818413734436, -0.5952812433242798, 1.5382565259933472, -0.6620542407035828, 0.9009760618209839, -1.3141249418258667, 0.6544262170791626, -0.288227379322052, -1.2239840030670166, 0.894029438495636, 1.178390622138977, -0.48730844259262085, -0.6273754835128784, -0.3921144902706146, -0.8957247138023376, 0.18922439217567444, 0.3102820813655853, 2.0847465991973877, -0.9131869673728943, -1.3314032554626465, -2.1218597888946533, 0.5970631241798401, -0.5821142792701721, 2.635199785232544, 0.7806163430213928, 0.2972460389137268, 0.30698099732398987, 0.906728208065033, -0.9434043169021606, -0.9616295695304871, -0.6070708632469177, -0.7076884508132935, -1.9431146383285522, -0.6568257212638855, 1.079292893409729, -0.005398237612098455, -0.742428719997406, -0.8198011517524719, 0.6311296820640564, 1.0217580795288086, 0.4271618723869324, 0.1517709344625473, 1.4974901676177979, -0.9574357867240906, 0.7299100756645203, -1.059509038925171, -1.9536640644073486, -0.08366158604621887, -1.1048320531845093, -0.7057914733886719, 1.3184260129928589, 0.4846278727054596, -0.06979323923587799, -1.6261764764785767, 2.010496139526367, 0.8932737708091736, 0.02629433386027813, -1.2574542760849, 0.39894381165504456, 0.3553190529346466, -0.007048454601317644, -1.0513758659362793, 0.5550018548965454, -0.004557210952043533, 1.2257881164550781, 0.5483360886573792, 1.2659404277801514, -0.8464130759239197, -1.5830310583114624, -0.2141890525817871, 0.9700949788093567, 0.7911161184310913, -0.4380389153957367, -0.20071819424629211, 0.4635419547557831, 1.9261325597763062, 0.3368721306324005, -0.587613582611084, 2.021968126296997, 0.47314128279685974, 1.5616534948349, -1.1886236667633057, -0.5220924615859985, 0.957203209400177, 1.2554837465286255, 0.43585941195487976, -1.3322036266326904, 1.0661321878433228, -0.39608705043792725, 0.43512046337127686, -1.9105035066604614, 0.49162155389785767, -0.6941478252410889, -0.168924018740654, 1.2878499031066895, 0.5256813764572144, 0.3332909941673279, -1.543737530708313, -3.2626264095306396, -1.818786859512329, -1.0105782747268677, 0.9087695479393005, 0.2974647283554077, -0.5257739424705505, -1.703945279121399, -0.2088681012392044, -0.5134625434875488, 0.3128288984298706, 0.8298307061195374, 0.267055481672287, -0.7788376808166504, 0.41825592517852783, -2.2963671684265137, -1.0397183895111084, 0.1644873321056366, 0.34872278571128845, -0.6343420743942261, -1.6565837860107422, -1.0286670923233032, 0.6956402659416199, -0.04978805407881737, -0.3877793848514557, 0.1426161676645279, -2.180881977081299, 0.4505878686904907, 1.2016023397445679, 3.08245587348938, 0.5945410132408142, -0.15428510308265686, 0.30483385920524597, -1.0529872179031372, -0.3363344073295593, -1.3246561288833618, 0.3102596402168274, -0.21015766263008118, 0.427447646856308, 2.2068538665771484, -0.9324162006378174, -0.2451624572277069, -1.1475363969802856, 0.33300766348838806, -1.0193113088607788, -0.6968713998794556, -1.0262547731399536, -1.9954171180725098, -0.9708734750747681, -0.10142964124679565, -0.7380697727203369, 0.7544226050376892, 0.06488465517759323, -0.3148563802242279, -0.28990986943244934, -0.11992336809635162, -0.27750611305236816, -0.25243106484413147, 0.6927834749221802, -0.6012113690376282, 0.3671016991138458, -0.2058323323726654, -0.16216078400611877, -1.4302526712417603, 0.09744895994663239, 0.8650642037391663, 0.8479416966438293, 2.232219934463501, -0.23516640067100525, -1.9735163450241089, 1.2335981130599976, -2.4683961868286133, -0.6572726964950562, 0.06412981450557709, -1.3887102603912354, 0.7985392808914185, -0.6496075987815857, 1.4381448030471802, -0.8749547004699707, 1.5087270736694336, 0.40210631489753723, 0.12662070989608765, -0.08261366188526154, -0.5904161930084229, -0.033125657588243484, -0.28916385769844055, -0.13891445100307465, -0.5156525373458862, 0.37499257922172546, -0.16486994922161102, 0.34190475940704346, 0.674723744392395, 1.0468684434890747, 0.6243541836738586, -1.3575003147125244, -0.17716403305530548, -0.8604421615600586, 0.550319254398346, 0.1265069544315338, -1.0042659044265747, 0.9673212766647339, -0.8334542512893677, -0.39932456612586975, 1.0719670057296753, -0.49388694763183594, -0.5266286730766296, -0.13452433049678802, 0.1600029468536377, 1.3408920764923096, -0.19762533903121948, -0.6743775010108948, 0.06161738559603691, 0.6824597120285034, -1.5405848026275635, 0.5648272633552551, -0.38588735461235046, -0.2136862874031067, -0.3581821918487549, 1.0396159887313843, 1.5086402893066406, -1.7409932613372803, 0.08054453879594803, -0.6794179081916809, -0.24300667643547058, 0.6564344763755798, 0.9257377982139587, -1.2017954587936401, -0.09000182151794434, 0.6600423455238342, -0.9489859938621521, -1.0891661643981934, 0.3703714609146118, 0.2433301955461502, 0.29520556330680847, -1.4966927766799927, -0.7742494940757751, 0.5831837058067322, 0.06641803681850433, 1.3743456602096558, 0.2900846600532532, -1.4125747680664062, 1.5217622518539429, -1.199374794960022, -0.875036895275116, -0.23685498535633087, 2.3138163089752197, -0.4959881603717804, 1.2199925184249878, 2.6842236518859863, -0.30863699316978455, 0.7174311280250549, 2.1107277870178223, 0.27599573135375977, -0.013429392129182816, 0.44771847128868103, -0.007463094778358936, 0.9810360670089722, -0.7834503650665283, 0.6987619400024414, -0.6244089007377625, -0.05893641337752342, 0.0809755027294159, -0.010657143779098988, 0.37470710277557373, 0.7149854302406311, 0.09058623015880585, 0.18505901098251343, -0.5246334671974182, 0.06936997920274734, 1.4621797800064087, 0.2904931604862213, 1.2132885456085205, -0.5864060521125793, 0.4365277886390686, 2.569868564605713, 0.6144687533378601, 1.2700556516647339, -0.4471457004547119, -1.0153510570526123, -2.6134467124938965, 0.13628193736076355, 0.25859683752059937, 1.015386939048767, -0.00022516149329021573, 0.8389538526535034, 1.0873658657073975, 0.2406459003686905, -0.7347215414047241, -0.2625965178012848, -0.6620063185691833, 0.6709030270576477, -0.5161855816841125, 0.4626387655735016, -0.07378550618886948, 0.6416305899620056, 0.23714682459831238, -0.21075499057769775, 1.8467403650283813, -0.9358301162719727, -1.1169323921203613, -0.6453016996383667, 1.1600525379180908, -1.4030694961547852, -0.45336341857910156, 0.9773195385932922, 1.053029179573059, -0.8846054077148438, -1.4105749130249023, 0.8772886395454407, -0.06827986240386963, -1.3368772268295288, -1.4371891021728516, -0.8743327856063843, 0.04421650245785713, 0.03575918823480606, -1.455519199371338, -0.26527300477027893, 1.8043378591537476, -2.2371981143951416, -0.7947899699211121, 0.9236530661582947, -0.2341490536928177, 1.755354642868042, -0.7039613127708435, 1.411043405532837, 0.4106386601924896, 0.8364450335502625, 0.5605918169021606, -1.3340774774551392, -0.06153789535164833, 0.012306069023907185, -0.2251279056072235, -1.2342439889907837, -0.24612219631671906, -0.2837066650390625, 0.15446537733078003, -0.2450748234987259, 1.7689961194992065, 0.07294684648513794, -2.1142334938049316, 1.1689482927322388, -0.7397730350494385, 1.5374927520751953, -1.0302555561065674, 0.28563791513442993, -0.3067939877510071, -1.0570895671844482, 0.37533479928970337, 1.3449506759643555, -0.05097788944840431, 0.63572758436203, 0.23322328925132751, -0.7146793007850647, -0.9556606411933899, 0.4975341856479645, -0.6483785510063171, -0.379578173160553, 0.3383707106113434, -0.5218475461006165, 0.026279570534825325, 0.39264848828315735, 1.0336796045303345, 1.7694032192230225, -0.8386572599411011, -0.35744908452033997, 0.16685138642787933, 1.2729068994522095, 0.30272161960601807, -0.3051524758338928, 0.42093154788017273, -1.4256983995437622, 0.5407896637916565, 0.06428619474172592, 1.4188957214355469, -1.5722541809082031, -0.14588329195976257, -0.461505651473999, 2.184103488922119, -0.09286827594041824, 0.38030004501342773, 0.9320275187492371, -0.363522469997406, 0.7044609189033508, -0.8289550542831421, 1.32411789894104, -1.6924359798431396, 0.3978753685951233, -1.2179328203201294, -1.118927001953125, 0.41373583674430847, -0.09499377757310867, -0.7394565343856812, -0.44233641028404236, 2.3398523330688477, 0.6690065264701843, -0.30619773268699646, -0.24354396760463715, 0.43716949224472046, -0.8682451844215393, -1.365919589996338, -0.6278774738311768, 0.035743892192840576, -0.15152673423290253, 0.2749707102775574, -2.018979549407959, -0.5012295246124268, 0.9868228435516357, 1.1741392612457275, 0.9356122612953186, 1.1319409608840942, 1.7113327980041504, 0.7876287698745728, -1.6993299722671509, 0.508560061454773, 1.244238257408142, 0.060628339648246765, -1.738043189048767, 0.04047302156686783, -1.7079766988754272, -0.725874125957489, -1.367363452911377, 2.6361777782440186, -0.31588950753211975, 0.3252106308937073, -0.000909823109395802, 0.20791542530059814, -0.1605140119791031, -0.31845641136169434, 0.24815990030765533, -0.8031092286109924, 0.4376749098300934, -0.6732715368270874, 0.7885411381721497, 0.7515317797660828, 0.3376845717430115, -1.196765422821045, 2.2806313037872314, 0.3672836720943451, -1.1099098920822144, 0.6305528283119202, -0.6122927665710449, 2.326920509338379, -1.4908385276794434, 0.03658470883965492, 0.7177299857139587, -0.7771767973899841, -2.1328909397125244, 0.36755111813545227, 0.5394049286842346, -0.8626906275749207, -0.5251182913780212, -0.5801503658294678, 1.2363680601119995, -0.6289670467376709], [1.4264942407608032, 1.1184247732162476, -0.18046768009662628, -0.9291832447052002, 1.22682785987854, -0.9452773332595825, 0.5160796046257019, -0.3967715799808502, 0.33796998858451843, 0.35284122824668884, -0.6790904998779297, 0.18510976433753967, 0.21196651458740234, 0.457570880651474, 1.2938463687896729, -1.1771156787872314, 0.4140385687351227, 1.9233053922653198, -0.39061596989631653, 0.26627710461616516, -1.3508063554763794, 0.17702649533748627, -1.0098438262939453, -0.9432839155197144, -0.7816208004951477, -0.9814221858978271, 1.1101131439208984, -0.16607898473739624, -0.3231760263442993, -0.39862388372421265, -0.8685138821601868, -0.2885349690914154, 1.3522201776504517, 1.6517982482910156, 0.8650500178337097, -0.08997499942779541, -0.048882246017456055, 0.5228540301322937, -0.3960922360420227, 0.41010186076164246, -0.4884948134422302, 0.9980453252792358, -0.7513958215713501, -0.9493356347084045, 0.7938240766525269, 0.34932559728622437, -0.7241432070732117, 1.8051866292953491, -0.5152840614318848, 0.3916594982147217, -0.6615486741065979, 1.374568223953247, -0.422148734331131, -0.05787886679172516, -2.4547321796417236, -1.7014232873916626, 0.15482929348945618, -0.16835057735443115, 2.3189194202423096, 1.4325124025344849, -1.078429937362671, 0.9232745170593262, -0.3614402413368225, -0.3445771038532257, -0.34297168254852295, 2.2257816791534424, -0.37188974022865295, -0.679456353187561, -0.5216276049613953, -1.7029508352279663, 0.5825846195220947, 0.038390375673770905, -0.5092735290527344, 1.0154547691345215, -0.5196934938430786, 0.4035956859588623, -0.2961014211177826, -0.8081399202346802, 0.07319726794958115, -1.0923386812210083, -0.40165984630584717, 2.062967538833618, -1.1314051151275635, -0.9145534634590149, -0.2621474266052246, 1.479151964187622, -1.0727041959762573, 1.0348137617111206, 0.514904260635376, 1.028076410293579, 0.9141599535942078, 0.730747640132904, 1.0930736064910889, -0.3504577577114105, 0.2129814624786377, 0.5164051651954651, -0.7884841561317444, 0.5520626306533813, 0.3748871982097626, -0.3185955286026001, 1.0162975788116455, -0.30078253149986267, 0.5534003376960754, 0.7202454805374146, 0.9482288360595703, -0.061485789716243744, -0.7116287350654602, 1.1629801988601685, 0.16313622891902924, -0.027549585327506065, 1.461015224456787, -0.9633706212043762, 0.6355727314949036, 0.38337796926498413, 0.04454563558101654, -0.12736749649047852, 2.171628475189209, -2.057985544204712, -0.5742169618606567, -0.22289784252643585, 0.11424222588539124, -0.9301638603210449, 0.23470793664455414, 1.2503074407577515, 0.43085381388664246, -0.18161505460739136, -1.1663682460784912, -0.5904883146286011, -1.2479647397994995, -0.17933832108974457, 0.16149063408374786, 1.5363701581954956, -0.8087486028671265, -0.29806217551231384, 0.5264630913734436, -1.6316349506378174, -1.152091383934021, -1.5074048042297363, -0.1166505292057991, -0.1435796320438385, 1.0036498308181763, 0.789941132068634, -0.9096450805664062, 0.44527918100357056, -0.791287362575531, -1.3995330333709717, 0.1404937207698822, -0.0026959639508277178, 1.3046517372131348, 0.14511270821094513, -0.4927022457122803, 0.6556403636932373, -0.16252747178077698, -2.8804705142974854, -0.895581841468811, -0.05672867223620415, 1.0721311569213867, 1.4425523281097412, 0.6245725750923157, -0.9052382707595825, 0.41063278913497925, -1.3061848878860474, -0.2568340003490448, -0.7921052575111389, 1.329136848449707, 1.0130220651626587, 0.9975523352622986, -0.019724421203136444, -0.9276437759399414, -1.6982722282409668, 0.42832159996032715, -1.5971599817276, 0.44630128145217896, -1.8777447938919067, -0.20894354581832886, -0.19884566962718964, 0.318766325712204, -0.9124220013618469, -0.2701977491378784, -0.19601091742515564, -1.3106117248535156, 0.3323865830898285, -1.016758918762207, 0.12731294333934784, 0.028717223554849625, 0.8836982846260071, 0.2633760869503021, -0.20170904695987701, -0.2088051587343216, 0.04334893450140953, -0.9434710144996643, 2.3445141315460205, 0.4990253150463104, 1.6014578342437744, 1.1826115846633911, -0.46398845314979553, 0.6177024841308594, 0.4038088321685791, 1.078731656074524, -0.7937653660774231, -1.0427533388137817, 1.0793429613113403, 0.5415045022964478, -0.43995344638824463, -1.2578339576721191, -1.8371142148971558, 1.8483426570892334, -0.18545858561992645, 0.37469568848609924, -1.0530166625976562, -2.74381685256958, 0.03577470779418945, 2.5702922344207764, 0.22855615615844727, -0.6047950387001038, -0.05642586201429367, -1.5329504013061523, -1.4751003980636597, -0.09798429161310196, 1.9997045993804932, 1.4572889804840088, 0.04802694544196129, -0.06609798967838287, -1.3627123832702637, 0.367427796125412, -0.5849592089653015, 0.49754154682159424, 0.5272759199142456, 0.46483153104782104, 1.0390578508377075, 0.34719118475914, 2.6564433574676514, -0.052748046815395355, 1.6310017108917236, -1.8903884887695312, 0.5118688941001892, 0.7085062265396118, 0.4257707893848419, -1.6798752546310425, -0.8328676223754883, 0.030060280114412308, -0.7145738005638123, -0.08841995149850845, -0.05839846655726433, -1.6148546934127808, 0.31178537011146545, 0.9890174269676208, -0.32805389165878296, -0.2993459105491638, -0.932859480381012, -1.6051290035247803, 0.5137721300125122, 1.7599129676818848, -0.33828282356262207, -0.5215921998023987, 1.94963800907135, 0.30593568086624146, 0.8166229724884033, -0.04558584466576576, -0.07738049328327179, -0.6243841052055359, 0.6782140135765076, -2.1714601516723633, -0.027001628652215004, 0.3808056116104126, 1.0351628065109253, 1.0783425569534302, -1.2688016891479492, 0.5119808316230774, -1.4272541999816895, 1.7908610105514526, -0.13451546430587769, 2.1688973903656006, 0.9842082262039185, -0.9529630541801453, -1.0405817031860352, 1.4756327867507935, -0.17911973595619202, -1.4642797708511353, -1.4849764108657837, -0.7229791283607483, -1.4219993352890015, -1.0784233808517456, -1.427153468132019, 0.7643487453460693, 1.2642197608947754, 0.7562924027442932, 1.5048332214355469, -0.7612453103065491, -0.9192983508110046, 1.5688531398773193, -0.03246273845434189, 0.5371598601341248, 1.1164675951004028, 0.15579275786876678, -2.4988865852355957, 1.09642493724823, 1.1477340459823608, -0.7155116200447083, -0.7934345602989197, 0.02439075894653797, -0.19970574975013733, 0.9096229076385498, 0.8578583002090454, -1.4095203876495361, -0.3451569676399231, -1.1820619106292725, -0.14895595610141754, 1.8168069124221802, -1.5332103967666626, 2.1415796279907227, -0.2027989625930786, -0.39827924966812134, 1.720309853553772, -0.28288671374320984, 1.362428069114685, -0.29478374123573303, -0.15102674067020416, 1.0447667837142944, -0.43351802229881287, 0.8155102729797363, -0.7922417521476746, 0.2665766179561615, 0.552706778049469, 0.7685790657997131, 0.5914644002914429, 0.49570536613464355, -0.7648520469665527, 1.432329773902893, 0.7034875750541687, 1.4558213949203491, -0.4448597729206085, 0.4643567204475403, 0.18464116752147675, -2.380967140197754, 0.4468366801738739, -1.113621473312378, -0.21972021460533142, 0.9964999556541443, -1.1637837886810303, -1.4805909395217896, 0.23325106501579285, -0.4442940354347229, 2.2700891494750977, 0.6147836446762085, 1.2405887842178345, -0.340032160282135, -1.6875501871109009, 1.6889036893844604, 0.9237991571426392, 0.7640534043312073, -0.657924234867096, 0.5642427802085876, 0.12689968943595886, -0.10522688925266266, 0.8642988801002502, 1.0689195394515991, -0.7605054974555969, 0.12983761727809906, 1.8101998567581177, 0.7647129893302917, -0.10658131539821625, -1.3535281419754028, 1.13861882686615, 2.202434539794922, -1.393043875694275, 0.5368461608886719, 0.9554338455200195, 0.05820276960730553, 1.2640854120254517, -0.800047755241394, 0.8961712121963501, -0.8728886842727661, 0.49564629793167114, 0.9720081090927124, -1.3370184898376465, -0.2823452651500702, 0.1678878664970398, -0.9874873161315918, 0.9695468544960022, 0.5972191691398621, -0.7800162434577942, -0.5480327010154724, 0.6400894522666931, -1.6383718252182007, 1.0307680368423462, 0.3543180525302887, -0.5218380689620972, -0.652018129825592, 0.2807198762893677, 0.569196343421936, 0.07814287394285202, 0.3214210271835327, 0.8086111545562744, 1.1917709112167358, -0.7239729762077332, -1.3279699087142944, -0.5746116042137146, -1.0463979244232178, 0.10496269166469574, -0.08370751142501831, -1.0031251907348633, -0.674692690372467, -0.7015673518180847, -0.9290086627006531, 2.3841261863708496, -0.5622115731239319, 0.9174000024795532, -0.9779999852180481, 1.4063431024551392, 0.9733848571777344, 1.5163835287094116, 1.0045669078826904, -0.893444836139679, 0.7095676064491272, -0.6440250277519226, 1.4048340320587158, 0.3594295382499695, -2.9674336910247803, 1.4049032926559448, -1.0942094326019287, -1.293111801147461, 1.19464910030365, -1.139326572418213, 1.8489714860916138, 0.4070940613746643, 0.25960424542427063, -2.0394809246063232, 1.4882771968841553, -2.3562567234039307, -1.5736607313156128, -0.8656765222549438, -0.29603490233421326, 0.0837632417678833, 0.018109334632754326, 0.7839221358299255, 0.5894257426261902, -0.9043030142784119, -0.9310626983642578, 0.19098296761512756, 0.6553248167037964, 1.9250794649124146, 0.04732302576303482, 2.2550039291381836, -0.08950717747211456, -0.7984707951545715, 0.8400040864944458, 0.6871213316917419, 2.092883586883545, -0.09331007301807404, -1.7119466066360474, 0.8533564209938049, 0.9032545685768127, -0.14158426225185394, -0.8422884941101074, 1.3168256282806396, 1.9090262651443481, -0.058296751230955124, 2.3876123428344727, 1.7686229944229126, 0.15363065898418427, -2.20503306388855, -0.5666617751121521, -1.1588478088378906, 0.1788340061903, -0.3002133071422577, 0.01617078296840191, -0.8454191088676453, -0.7724297642707825, -0.6212497353553772, 0.8040733933448792, -0.7098025679588318, -0.3937605917453766, -1.3551559448242188, 0.11871657520532608, 1.4216312170028687, -0.27677467465400696, -1.7908365726470947, -1.392576813697815, -0.8326690196990967, -1.1931405067443848, -0.9969770312309265, 2.09896183013916, 0.5316163301467896, 0.29802608489990234, 1.2338734865188599, 1.4270763397216797, -0.6235477924346924, 0.3955327272415161, 1.3425215482711792, -0.2663685381412506, 1.3389441967010498, 1.2335751056671143, -0.7040508985519409, -1.2187780141830444, 0.43005144596099854, 0.052767835557460785, -1.3305201530456543, 1.2455741167068481, -0.27696919441223145, 1.0299572944641113, 0.23811016976833344, 0.8017900586128235, 0.10554666817188263, 1.1201924085617065, 0.6943846940994263, 0.7204849720001221, 1.8560847043991089, -1.857350468635559, -1.1945923566818237, 0.4836118519306183, 0.03187277913093567, -0.6119711995124817, 1.1677273511886597, -1.683472990989685, 0.05229639634490013, 0.8023909330368042, -0.8089919090270996, -1.0861190557479858, 0.8181121349334717, -0.04349517822265625, -1.0288524627685547, 1.1661155223846436, -1.7184759378433228, 1.4112510681152344, -1.1433392763137817, -0.8815861940383911, -1.3308173418045044, -0.14307457208633423, 0.3322029709815979, -0.7148492336273193, -0.2609065771102905, -0.8569874167442322, 1.0111390352249146, 0.25269827246665955, -0.4977707266807556, -0.9054200053215027, 0.860808253288269, -0.03580346703529358, -0.3612893521785736, -1.790727138519287, -2.692260265350342, 0.6431943774223328, -1.2720426321029663, -0.4159565269947052, 1.0327177047729492, 0.5913977026939392, 0.9944490194320679, 1.6362323760986328, 0.872909426689148, -2.117551326751709, -0.7620300054550171, -0.8211366534233093, -0.4821580648422241, 1.024414300918579, 1.4541833400726318, 0.09904755651950836, 1.7533868551254272, -1.7509697675704956, -0.013152704574167728, 1.442326545715332, -0.3240258991718292, -0.214504212141037, -2.144590139389038, 0.5512372851371765, 0.7694898247718811, -0.11109249293804169, -0.12730804085731506, -1.8144241571426392, 0.0956631451845169, 1.2732441425323486, -0.06401649862527847, 0.31662026047706604, -0.42920947074890137, 1.4393223524093628, -0.3875857889652252, 0.39687877893447876, -3.313084363937378, 0.2555360198020935, -2.510923147201538, 1.863588809967041, 0.785857617855072, 0.3660515248775482, -1.5183374881744385, 2.7924511432647705, -2.215437412261963, -0.09946011006832123, 2.296299695968628, -0.14551351964473724, 0.9156906604766846, -0.2923426628112793, 1.749560832977295, -0.6918567419052124, 2.467581033706665, 0.3428714871406555, -0.07550317794084549, -0.18664175271987915, 0.3451682925224304, -0.24182669818401337, -0.25659552216529846, -1.3699487447738647, 0.5750144720077515, 0.48758432269096375, -1.2299785614013672, -0.6046618819236755, 1.152976155281067, -1.3385705947875977, 1.0237706899642944, -0.30421167612075806, -0.6189044117927551, 0.7329042553901672, -0.38283905386924744, -0.2812223434448242, -0.6450624465942383, -0.5943414568901062, -1.9174716472625732, -0.4867979884147644, 0.6330894231796265, -0.9031258225440979, -0.9018630385398865, 0.3522825837135315, 0.6684731245040894, -2.09554123878479, 0.9976670742034912, -0.868176281452179, -0.7993592023849487, -1.0529998540878296, 1.0692083835601807, -0.2970027029514313, 0.8358848690986633, -0.8075045943260193, 0.947689414024353, -0.19788387417793274, 0.013728396035730839, -0.5995648503303528, -1.4825257062911987, -0.8353382349014282, -0.6848273277282715, -0.47985514998435974, 0.6433486938476562, 1.1207224130630493, 1.405889630317688, 0.19399510324001312, 1.2362834215164185, -0.09235075861215591, -1.4767659902572632, 1.0802713632583618, 0.0023468518629670143, -1.8839111328125, -2.1422266960144043, -0.36886563897132874, -0.8391371369361877, 1.1637722253799438, -2.4919016361236572, 1.284583568572998, -0.11013847589492798, -0.13810119032859802, -0.010381192900240421, 0.010901344008743763, -0.06348082423210144, -0.25284281373023987, -1.0213514566421509, 1.1303448677062988, 0.025896653532981873, -0.02524080127477646, 0.09616348892450333, 0.8182929158210754, 0.9376140236854553, -0.6389124393463135, -0.054521799087524414, -0.07620903849601746, 0.8282813429832458, -0.8872337937355042, 1.8958443403244019, -0.835314154624939, 0.8408109545707703, -0.5361030697822571, -1.3261584043502808, -0.221106618642807, -2.2227776050567627, 1.9742451906204224, -0.12050295621156693, -0.3189561665058136, -1.2149081230163574, -2.191577672958374, 0.577450156211853, 0.5185383558273315, -0.20077645778656006, 0.3010138273239136, 1.121142864227295, -0.384289026260376, -1.0646498203277588, -0.8324323892593384, -1.2225658893585205, 0.6872588396072388, -1.0352839231491089, -0.6901053190231323, 1.2103019952774048, -0.8980655074119568, -0.20029564201831818, -0.01773037575185299, -0.37352555990219116, 0.039363086223602295, -1.0736252069473267, -0.4688156247138977, -1.0567935705184937, -1.0034692287445068, -0.44499868154525757, 1.0607939958572388, -1.1917378902435303, -0.5443378686904907, 0.3001015782356262, 0.0031774018425494432, 0.23721005022525787, 1.2019541263580322, 0.5242545008659363, -0.057449132204055786, -0.07537548243999481, -0.9488542675971985, -1.0074396133422852, -0.04411783814430237, -0.2114439159631729, 1.0906773805618286, -1.1690044403076172, -0.3675169050693512, -0.5221274495124817, 0.09272056818008423, -0.32507386803627014, 1.3205795288085938, 0.554015040397644, -1.2417054176330566, -0.5680283308029175, 0.3742469847202301, 0.0017074738861992955, -0.9224187135696411, -0.024489380419254303, -0.01108672097325325, -1.2946984767913818, -1.3217356204986572, -0.08835823088884354, -1.3537160158157349, -0.14417634904384613, 1.3324931859970093, -1.0122952461242676, 0.3564891815185547, -0.6193549633026123, -0.3370126485824585, 1.016348123550415, -1.2327080965042114, 0.4899643659591675, 0.4296848773956299, -0.12753213942050934, -0.3665485680103302, -0.1829562485218048, 1.0037480592727661, -1.4198646545410156, 0.1107376366853714, -0.4454110562801361, -1.3705543279647827, 0.04402649402618408, -0.4098649024963379, 0.29031991958618164, 1.1299816370010376, 0.31689170002937317, 0.5495415925979614, 0.563284158706665, -0.010282863862812519, -0.2635573148727417, -0.011923783458769321, 0.8510234951972961, 0.4085887670516968, 0.28279727697372437, 0.1327468603849411, 0.7848587036132812, -1.8056024312973022, -0.8510732650756836, -1.8695547580718994, 0.2782120704650879, -0.338957279920578, 0.8098195195198059, -0.7708456516265869, 0.5844637155532837, 0.5897610783576965, 1.0422921180725098, -0.15454792976379395, 0.9028040766716003, 0.044235844165086746, -0.2607489824295044, 0.7285963892936707, -0.39951419830322266, -1.085607647895813, -1.4625496864318848, -0.41213202476501465, 2.0076513290405273, -1.2680234909057617, 1.7993744611740112, -0.6930562257766724, 0.6795450448989868, -0.40019163489341736, 0.16440773010253906, -1.032489538192749, 1.9574030637741089, -0.4110865890979767, 0.7929258942604065, -0.8157908916473389, 0.7095337510108948, 0.05390683934092522, 1.9961743354797363, -1.35896635055542, 0.36142784357070923, 1.8547264337539673, -0.06129668653011322, 0.3499261438846588, -0.8588577508926392, -0.6682235598564148, -2.530402660369873, -0.8034008741378784, -0.33026349544525146, 0.7430466413497925, -1.118174433708191, 1.0237467288970947, -0.3500773012638092, 1.0777881145477295, 1.696772813796997, -2.360072374343872, -0.28867900371551514, 1.581839919090271, -0.22747118771076202, 0.1110931932926178, 0.3133241534233093, 0.02366851270198822, -0.8981786370277405, 1.3064559698104858, 1.5870733261108398, 2.350224256515503, 0.6766451001167297, -1.4232836961746216, -1.9120649099349976, 0.8192920684814453, -0.21038316190242767, 0.2151637077331543, -1.342753529548645, 0.09958437830209732, 0.1063818633556366, -0.16569587588310242, 1.5745044946670532, -1.1024973392486572, -1.0890990495681763, 0.3081694543361664, 0.8521048426628113, 0.5202507972717285, 0.36733561754226685, -0.5976659059524536, -0.9834303259849548, -0.006147945765405893, 0.39376068115234375, 0.7330445051193237, -0.01752244494855404, -0.8704454898834229, -0.8389400243759155, 0.6566027998924255, 0.9511702656745911, 0.6829986572265625, 1.0725834369659424, -0.654991865158081, 0.018417222425341606, 1.1898974180221558, -0.6338960528373718, 0.06742385774850845, -0.35008394718170166, 0.5920054316520691, 1.1001461744308472, 1.8797775506973267, -0.37250277400016785, -1.2966296672821045, -0.22869765758514404, -0.36124396324157715, 0.37155285477638245, 0.386732280254364, -1.1578190326690674, 0.8609400987625122, 1.1828064918518066, -0.03295620530843735, 1.403961181640625, -0.014588222838938236, -0.2552626132965088, -1.1151992082595825, -0.6859086751937866, 0.05712030082941055, -0.28387606143951416, -0.19711218774318695, -0.32646578550338745, 1.025458574295044, -1.210829734802246, -0.4548596143722534, -0.17666198313236237, 0.8773533701896667, -2.705158233642578, 0.7405861020088196, -0.8236919045448303, 0.7938476800918579, 0.35775208473205566, 0.03266267105937004, -0.14662353694438934, -1.518027663230896, 0.5507904887199402, -0.3668113946914673, 0.5853376388549805, -0.13063102960586548, -0.6868148446083069, 0.7422273755073547, 1.6162745952606201, -0.7557389736175537, 0.110965795814991, -0.878910481929779, 1.6113041639328003, 0.07580425590276718, 1.6301974058151245, 1.1998744010925293, 1.1911853551864624, 1.8352034091949463, 1.9886853694915771, -0.6819711327552795, -0.7398157119750977, 0.5536672472953796, -0.6430990695953369, 0.913678765296936, -0.9677015542984009, -1.3555654287338257, -1.1301966905593872, 1.260444164276123, -0.3553315997123718, 0.502754807472229, 0.8197412490844727, 0.2262953370809555, 2.0546672344207764, -0.8331142067909241, -1.1090596914291382, 1.2064203023910522, 0.17182329297065735, -0.6887592077255249, 2.205650806427002, -1.1561557054519653, -0.6266200542449951, 1.5869451761245728, -1.3427239656448364, -1.5787811279296875, -1.3457049131393433, -1.1729825735092163, -0.8402249217033386, 0.5294729471206665, 1.08595871925354, -0.0646207183599472, -0.7361819744110107, 2.8540396690368652, -0.1713157445192337, -0.3746463656425476, 0.13981345295906067, 1.0096279382705688, 0.472778856754303, -0.28700608015060425, 1.114249587059021, -1.7599077224731445, 0.15020470321178436, -0.28090837597846985, 0.8491753339767456, 0.493745356798172, -0.46867838501930237, -0.691473126411438, -1.8491109609603882, -0.8402338624000549, -1.4123767614364624, 0.9425145983695984, 2.148409605026245, 1.1776021718978882, 0.14255234599113464, -0.8111215829849243, -0.3836280107498169, 0.5869338512420654, 0.5197199583053589, -0.09626728296279907, -0.3533076345920563, -0.42159491777420044, 1.1516361236572266, -1.5916540622711182, 2.064880847930908, 0.20138335227966309, 0.7601072788238525, 0.9491015672683716, -0.43546542525291443, -1.5069684982299805, 0.07219453155994415, 0.3728395104408264, -0.3493782877922058, -0.19965049624443054, 0.270272821187973, 0.2538597285747528, 0.9128124117851257, -0.12320395559072495, -0.6630704998970032, -1.5491665601730347, -0.11345812678337097, 1.6165968179702759, 0.5277562737464905, -1.2326871156692505, -1.2689685821533203, -0.04784523695707321, -0.4156631529331207, -1.0485353469848633, 0.6501994729042053, 1.8158164024353027, -0.24939163029193878, -0.8818459510803223, -0.2079029530286789, -1.8460369110107422, -0.10438960045576096, -0.7321975231170654, -0.22558651864528656, -0.0034046878572553396, -0.17650529742240906, -1.8404258489608765, -0.6271648406982422, -0.8890739679336548, -0.5698620676994324], [-0.39645031094551086, 0.5492130517959595, -0.14528964459896088, 1.1132720708847046, 1.1986593008041382, -0.742526650428772, 1.3648008108139038, 0.7895257472991943, 0.14974962174892426, -1.2390230894088745, -0.05834679305553436, -1.2515544891357422, -0.2523164749145508, -0.6500837802886963, 1.108330488204956, -0.8639835715293884, 0.18118160963058472, -1.4911741018295288, -0.9596842527389526, 0.6357715129852295, -0.9714404344558716, 1.0707095861434937, 1.1200320720672607, -0.09130870550870895, -0.5316268801689148, -0.8413863778114319, 1.4249401092529297, 0.45073601603507996, 1.126564621925354, -1.202929973602295, -0.3536367118358612, 2.2175371646881104, -0.1287539005279541, 0.5537773966789246, -0.3646698296070099, -0.11502879112958908, 0.4109836220741272, -1.1096916198730469, 0.3775304853916168, -0.8107844591140747, -0.1154278889298439, -0.30058470368385315, -0.4645392894744873, -0.12238345295190811, -0.5926287770271301, 1.2642905712127686, 1.4333736896514893, -1.4187248945236206, 0.8849788904190063, 1.192953109741211, -1.0897252559661865, 0.2805841565132141, 0.30199626088142395, 1.6296230554580688, 0.60249263048172, 0.533531904220581, 0.46880966424942017, -0.4219515919685364, -0.9581554532051086, -0.4387662410736084, -0.9629703164100647, 0.16771990060806274, 1.1306045055389404, -1.0469642877578735, 0.6189066171646118, 0.6590465903282166, -0.5233936309814453, -0.5468727350234985, 1.2057689428329468, -1.412718415260315, 0.7635157704353333, 0.053080715239048004, -0.8941360116004944, -0.5930172801017761, 0.36220383644104004, 1.1647934913635254, 0.5032444596290588, -1.3251063823699951, -0.24163001775741577, -0.11209449172019958, -0.3291458785533905, 0.38980746269226074, 0.33685654401779175, 2.370654344558716, -0.02814498543739319, 0.1406395584344864, 0.9390367269515991, 0.19426217675209045, 0.9297133684158325, 0.08438513427972794, -1.1529549360275269, 0.6074601411819458, -1.8725254535675049, -0.8230430483818054, 1.5640790462493896, 1.856397271156311, 0.728932797908783, -0.47558003664016724, -1.5518198013305664, -0.18470673263072968, -0.6424617767333984, -0.830943763256073, -0.7488014101982117, -0.01707521826028824, -0.3473045825958252, 0.12079745531082153, 0.5552940368652344, 0.7808708548545837, 0.13326570391654968, 1.9505294561386108, -0.20857444405555725, 0.8929029107093811, 0.08044496923685074, 0.8801066279411316, -0.9139994382858276, -1.321049451828003, 0.8923355937004089, 0.9137294292449951, 0.4030780792236328, 0.5705252885818481, 0.6508774161338806, 1.2209985256195068, 0.11558756232261658, -1.9976727962493896, 0.17543110251426697, 0.44680318236351013, 1.4080663919448853, -0.5393515825271606, 0.7357034087181091, 1.3747893571853638, -0.33454155921936035, 1.1085491180419922, 0.8192980885505676, 0.20505796372890472, 0.2427268773317337, 0.5834954380989075, -1.4671189785003662, -1.5530071258544922, 0.3989747166633606, -1.209475040435791, 1.1922129392623901, -0.32777443528175354, 0.0933079645037651, -1.0370711088180542, -0.1658271849155426, 0.1405915468931198, 0.36318057775497437, 0.6341474652290344, -1.005211353302002, 0.5816031098365784, -1.390075922012329, 2.153362512588501, -0.2841528356075287, -1.742387294769287, -0.8692440390586853, 1.7155691385269165, 1.7363014221191406, 0.7904127240180969, 1.1308400630950928, -0.10817531496286392, -0.8709728121757507, 0.4585261642932892, -0.08498620241880417, 0.9596880078315735, -0.8456258773803711, 0.939839243888855, -0.282257080078125, -0.12690389156341553, 0.636902928352356, 1.7444114685058594, -0.12804561853408813, -0.917646586894989, -0.8826090693473816, -0.19566549360752106, -0.37480220198631287, 0.49212679266929626, -0.4003957211971283, -0.07411527633666992, 0.6125476956367493, -0.4668651521205902, 1.8705979585647583, 0.7967404723167419, 1.159532070159912, -0.554919421672821, 2.3223981857299805, -0.5035817623138428, 2.810051918029785, -0.38900473713874817, -1.6212340593338013, 0.28567764163017273, 0.6011181473731995, -0.22666095197200775, 2.1944656372070312, -1.1436375379562378, -0.597927987575531, -1.1067256927490234, 1.1512089967727661, -1.7538998126983643, 0.6326848864555359, 2.1316370964050293, -0.2954513132572174, 0.035876449197530746, -0.2588347792625427, 0.9428225159645081, 0.9544034004211426, -1.8180612325668335, 1.538150429725647, 1.0018459558486938, -0.10640817880630493, -0.22333139181137085, -1.3465633392333984, 1.4129213094711304, -1.383538842201233, -0.5013415813446045, 1.6206190586090088, -0.31227484345436096, 2.248483180999756, -0.38659772276878357, -0.3882885277271271, -1.2099530696868896, -0.6062347292900085, -0.42438220977783203, -1.591549038887024, -0.05624709650874138, 1.0221657752990723, 0.6176489591598511, 0.6787427663803101, -1.2497665882110596, 0.0990849956870079, -0.04126862436532974, -2.0922837257385254, 0.98493492603302, -1.4084384441375732, 3.165649175643921, 0.1259811520576477, -0.7900355458259583, 1.1120126247406006, 0.2430770844221115, -0.20399577915668488, 0.031330741941928864, -0.6774097084999084, -1.6362978219985962, 0.5557113885879517, -0.34924590587615967, -1.6283657550811768, 1.3564225435256958, 1.032677173614502, -0.4676460027694702, 0.10263225436210632, 0.6699365973472595, 0.4365784227848053, -1.3556350469589233, 1.0127077102661133, 0.6879789233207703, -0.5248732566833496, 0.3386780023574829, -0.46330806612968445, -0.03019815869629383, 0.4798453152179718, -0.05468332767486572, 1.2124519348144531, 0.804392397403717, -0.7427417635917664, 0.08195916563272476, 0.9253753423690796, 0.864287793636322, -1.0852158069610596, -0.11775746941566467, -0.48791131377220154, -0.550576388835907, 0.4031769931316376, 0.4901205897331238, 1.9730943441390991, 0.18639418482780457, 0.34864500164985657, 0.5885576009750366, -0.6963815093040466, -0.9341223239898682, -1.1414064168930054, -0.9925357699394226, -0.5251155495643616, 0.16500461101531982, -0.8422103524208069, -1.3464796543121338, -1.327476978302002, 0.3039843440055847, 1.0860446691513062, 0.05642244219779968, -0.07646864652633667, 0.7114324569702148, -0.35748788714408875, -0.42929807305336, -0.7400350570678711, -0.8756574988365173, -0.3900172710418701, -0.805566132068634, 1.4427950382232666, -0.3125726580619812, 0.2699436843395233, 2.0736875534057617, -1.3142274618148804, -0.3004293441772461, -0.5315630435943604, -0.06325942277908325, 0.5855519771575928, -0.0759882777929306, -1.0913844108581543, 0.9070740938186646, 0.04432166740298271, 1.5732495784759521, -0.8513253927230835, -1.0049488544464111, 0.22231349349021912, -1.9932202100753784, -1.8341145515441895, -0.10922669619321823, -0.7074407339096069, -1.1487140655517578, 0.027872126549482346, -0.4571312963962555, -0.43186789751052856, -0.0918382853269577, 0.6557841300964355, 0.12977421283721924, 1.5678554773330688, -0.9396092295646667, 0.5614542961120605, 0.5054459571838379, -0.32072511315345764, 0.6663568615913391, 0.4160749018192291, -0.8722881078720093, -0.6534460186958313, 0.8113988041877747, -0.4649362564086914, 0.33447426557540894, 1.268953800201416, -0.3117217719554901, -0.8901998400688171, 1.7750097513198853, 0.23142854869365692, -0.03808644041419029, -0.3840790390968323, 0.2077220380306244, -0.4120057225227356, -0.9170845746994019, 0.5125770568847656, -0.05162737891077995, -0.48904746770858765, -0.229051873087883, 1.6820683479309082, 0.18974551558494568, -0.6862925291061401, 1.3236119747161865, -0.29714491963386536, 0.34591835737228394, 0.5578383207321167, -0.5387794375419617, 0.07780428230762482, -0.24244636297225952, 0.08923358470201492, -0.13835842907428741, -1.353564739227295, -1.1562446355819702, -0.1856994777917862, 0.8924546837806702, 2.016967296600342, -1.625099778175354, 0.8973401188850403, -0.09435983002185822, -0.4872612953186035, 0.9244346618652344, -2.471836566925049, -1.1947517395019531, -1.506769061088562, -0.23480863869190216, -1.3813978433609009, -1.5573885440826416, -1.078066110610962, 0.9797219634056091, 0.6602218151092529, -0.5090864300727844, -0.09366369247436523, 0.318859726190567, -0.33955830335617065, -1.0034301280975342, -1.6491814851760864, -2.38604474067688, -0.020767057314515114, 1.0426735877990723, -1.6849136352539062, 0.9655938148498535, 1.3641444444656372, -0.3015776574611664, 0.838982105255127, -1.3630387783050537, -0.6412122845649719, -1.1424843072891235, 0.7880368232727051, 0.6053966879844666, 1.0432482957839966, -0.4757355749607086, -1.7494395971298218, -0.5802887082099915, -0.08405684679746628, -1.2336692810058594, 0.3813773989677429, -1.6899352073669434, -0.4606056213378906, 0.28721994161605835, -2.002798318862915, 1.0045506954193115, 0.89267498254776, -0.35759419202804565, 1.357446551322937, 0.7919409275054932, -0.4100128412246704, 0.1802663505077362, 1.0467398166656494, -2.1586532592773438, -1.696115255355835, -0.06703940033912659, -0.9242838621139526, -0.7140224575996399, -2.747032880783081, 0.09256722033023834, 0.038778599351644516, -0.5571607947349548, 1.5874675512313843, 1.302391529083252, 1.0122162103652954, 1.6083959341049194, 0.856957197189331, 0.22578901052474976, 0.5530712604522705, 0.16282762587070465, -0.1381789743900299, 3.1581361293792725, -0.1723903864622116, 0.39448171854019165, -1.1847012042999268, 0.8139855861663818, 0.43908417224884033, 1.0501742362976074, -0.44459784030914307, 0.15743425488471985, -0.02093365788459778, -1.530800223350525, 1.6027508974075317, -0.17372764647006989, -0.5332112312316895, -1.5782026052474976, -0.8510805368423462, -0.6369795203208923, 0.5164605379104614, 1.1850496530532837, -1.3988237380981445, -1.3310387134552002, -0.9402327537536621, 0.46002197265625, 1.4237933158874512, 0.7188181281089783, -0.04861215129494667, 0.49816179275512695, 1.7976021766662598, -2.27181339263916, 0.11238493025302887, -0.757436215877533, -0.331503689289093, -1.243804931640625, 1.1534992456436157, 0.8215140104293823, 0.3013107180595398, -0.16106966137886047, -1.581666350364685, 1.4436153173446655, 0.47345665097236633, 0.39702704548835754, 1.231614351272583, 1.3840620517730713, 1.2341008186340332, -1.3418517112731934, 0.24362823367118835, 0.6075524091720581, -0.9634401202201843, -1.2860522270202637, -0.1718013733625412, -0.8081744909286499, 1.362360954284668, 1.1164698600769043, -0.6878491640090942, -0.7833967208862305, 0.19678382575511932, 1.2192270755767822, -0.9678661823272705, -1.7047713994979858, -0.41477495431900024, -1.2677711248397827, 0.21769379079341888, -0.6465427875518799, 0.8212308287620544, -0.052928291261196136, 1.5645525455474854, 0.19359040260314941, 1.4013460874557495, 0.7292447686195374, -0.8985817432403564, -0.8395797610282898, -0.2612776756286621, 0.07718057930469513, 1.0612443685531616, 2.0416057109832764, 0.5777148008346558, 1.216781497001648, 0.576208770275116, 1.6978548765182495, -2.132312297821045, -0.7553175687789917, 0.5043655037879944, -1.2780485153198242, 0.339044451713562, 0.6257465481758118, 0.12374351918697357, 1.158097267150879, -0.26923081278800964, 1.0463988780975342, 0.9329409599304199, 0.4505276083946228, -0.647523045539856, 0.2774006128311157, 1.1714346408843994, 1.1274665594100952, -0.19935746490955353, 1.5101343393325806, -0.7420483231544495, 0.848122239112854, 0.21757163107395172, -0.8715149164199829, 0.2980741560459137, -0.7877761125564575, 0.3721020817756653, 0.33989474177360535, -0.08714107424020767, 0.5297812819480896, 0.16749519109725952, 2.2138161659240723, -0.1442771553993225, -0.3803394138813019, 0.42082396149635315, 0.7138549089431763, 0.7313168048858643, 1.7797651290893555, 0.5213001370429993, 0.9896146655082703, -1.2281858921051025, 0.6844287514686584, 1.1741007566452026, -1.413504958152771, -0.20875029265880585, -0.3430539071559906, 0.9489337801933289, 0.2028745412826538, 1.3922297954559326, 0.6408892273902893, 0.850761353969574, 0.35954543948173523, 0.974608838558197, -0.14144586026668549, 0.4591984152793884, 1.979005217552185, -0.3824973404407501, 0.8400780558586121, -1.6267412900924683, -0.48358502984046936, -0.761345386505127, 0.12417594343423843, -1.7173538208007812, -1.5307329893112183, -0.13842204213142395, -0.539100170135498, 0.3587137460708618, -0.06916975229978561, -3.8219356536865234, 1.7538785934448242, -0.7046995759010315, 1.2759469747543335, -0.41813769936561584, -0.6799141764640808, -1.412741780281067, -0.4366479814052582, 1.199461817741394, -1.1222597360610962, -0.10513061285018921, 1.8297231197357178, -0.7141929268836975, 0.29981833696365356, -0.8550165891647339, -0.304271936416626, -1.0316829681396484, -0.9530996680259705, -0.9576520919799805, 1.125178575515747, 0.33033955097198486, -2.0728588104248047, -1.2280751466751099, 0.8056696057319641, 0.27753931283950806, -1.672458291053772, -0.66893070936203, 1.416014313697815, 0.17731699347496033, 0.39694610238075256, 1.6168038845062256, -0.9221972823143005, -2.162733793258667, -0.6639387607574463, -0.9280928373336792, 0.8396613597869873, -1.9281103610992432, 0.38029274344444275, -0.5609992742538452, -0.16902443766593933, -0.10010150820016861, -0.8613369464874268, -0.6429478526115417, -0.44111549854278564, -0.07230261713266373, -0.7739648222923279, -1.2848459482192993, 0.511448085308075, -0.9075627326965332, -2.3507614135742188, 1.0416494607925415, -0.274074524641037, -0.7546648979187012, 0.6953516006469727, -0.0019704033620655537, -0.45813918113708496, -1.5596650838851929, 0.023899901658296585, 0.7425159811973572, -0.3476867079734802, 1.1119705438613892, -0.3545050024986267, 1.2114017009735107, -1.5048688650131226, -0.4386252760887146, 2.835690975189209, -0.5495272874832153, -0.6923713684082031, -0.9825278520584106, -0.19166596233844757, -0.8171570301055908, 0.045526716858148575, -0.08104295283555984, 1.0767230987548828, -0.21220365166664124, -0.0465092658996582, 0.4097703695297241, 0.3971156179904938, 1.7735306024551392, -0.901151716709137, 2.2286179065704346, -1.0052106380462646, 0.4671352803707123, 1.3618581295013428, -0.153407022356987, -0.33739811182022095, 0.2073299139738083, -0.7388879060745239, 1.29149329662323, -1.8183623552322388, -0.36224544048309326, -0.6965970993041992, 1.5545504093170166, -2.031855821609497, -0.12101046741008759, 0.3641538918018341, -0.23492752015590668, 1.4681370258331299, -1.687798261642456, -0.2798781096935272, -0.4189579486846924, -1.5538413524627686, 0.5336160659790039, 0.4562152624130249, -0.12660282850265503, 0.5620209574699402, 1.4453189373016357, 0.8737095594406128, 0.9820244908332825, -0.8777232766151428, 2.097971200942993, 1.2989072799682617, -0.295492947101593, 0.20868517458438873, -0.1960262656211853, -0.5738692283630371, 0.25268107652664185, -0.01844482123851776, 1.2634965181350708, -0.09783283621072769, -0.06969398260116577, -0.6737746000289917, 1.1933505535125732, 0.8400052189826965, -0.9472784996032715, 0.1720348447561264, -0.4103001654148102, 0.5850977301597595, 0.7810993194580078, 0.019953006878495216, -0.6254644989967346, 0.4990156888961792, 0.3141724765300751, 0.18890216946601868, 0.33731403946876526, -0.17441929876804352, 0.6159308552742004, 1.3383092880249023, -1.5519380569458008, 1.3199443817138672, 0.2613472640514374, 0.20245593786239624, -1.392608404159546, 1.0908979177474976, -1.7738481760025024, -1.1809499263763428, 1.2391626834869385, 0.04637020453810692, 1.2968369722366333, 0.5896996259689331, 0.3956659436225891, 1.5167155265808105, -1.1542930603027344, 1.8631418943405151, -0.796188235282898, 1.2213693857192993, 1.6737326383590698, -1.2731654644012451, 1.4591463804244995, -0.7853009104728699, -0.2715539038181305, 0.6750733852386475, 1.3055245876312256, 1.0343834161758423, -0.5401856899261475, 0.5806162357330322, -0.711192786693573, 0.255881667137146, 0.7091155052185059, 0.36809995770454407, 0.6967955231666565, -1.8334462642669678, 2.072211742401123, 0.2704520523548126, 0.6536009311676025, 0.8759011626243591, 0.4228105843067169, -0.4916257858276367, 0.3526814579963684, 1.1785019636154175, -1.4836605787277222, 1.0375547409057617, -1.1955084800720215, 0.49101752042770386, -1.0386022329330444, 1.2510507106781006, -0.42179545760154724, -0.14783859252929688, -0.9169468283653259, -1.2337533235549927, 1.1465004682540894, 1.6799659729003906, -1.2316361665725708, -0.22343534231185913, 0.3146304488182068, -0.7325780391693115, 1.024956226348877, -1.1283642053604126, -0.41771745681762695, -0.7613976001739502, -0.18675167858600616, 0.059891410171985626, -0.574658215045929, 0.23954574763774872, -0.12308220565319061, 0.8044721484184265, -0.5051687359809875, -0.1951981484889984, 0.015091241337358952, 0.19142332673072815, 0.5669933557510376, -0.11245213449001312, 0.8817008137702942, -0.16873934864997864, -0.8016350269317627, -0.6613375544548035, -2.391846179962158, -0.08191566169261932, -1.6237794160842896, 1.0515843629837036, 2.049323081970215, 0.06902732700109482, 0.8953165411949158, -0.8413655161857605, 0.952322244644165, -0.73065185546875, -0.5451279282569885, 1.022965431213379, -2.1621954441070557, -1.8346800804138184, -0.9632826447486877, 0.6095569729804993, 0.9066236615180969, -0.2894754707813263, -2.510024309158325, -1.149600863456726, -2.176478862762451, 0.5449708700180054, -2.3508594036102295, 0.8211447596549988, -0.8198406100273132, -0.20853301882743835, -0.4405955672264099, -1.316191554069519, 0.8687641620635986, 0.018578162416815758, -0.18730752170085907, 0.34569603204727173, -0.8503777980804443, -0.18201011419296265, -0.03023335337638855, -0.3723316788673401, 0.5455471873283386, -1.0704270601272583, -0.9788891077041626, 1.0202723741531372, 0.9053348898887634, 3.1479086875915527, 0.5899162888526917, -0.9618136882781982, -1.0576140880584717, 0.23647651076316833, -0.469858318567276, -0.49691540002822876, -0.7422515153884888, -0.17616456747055054, -0.8723233342170715, -0.062275905162096024, -0.8095799088478088, -0.23770834505558014, -0.8154565095901489, 0.38501811027526855, 1.9567670822143555, -0.19960470497608185, -0.22869689762592316, -0.6005638837814331, -0.5286347270011902, -0.06502602994441986, -1.3084523677825928, 0.35222378373146057, 1.6743334531784058, -0.42587265372276306, -0.39007291197776794, -1.1117405891418457, 0.9646915793418884, -0.7589995265007019, 0.09155413508415222, -0.37886348366737366, 0.26760584115982056, -2.4134180545806885, -1.2382946014404297, -0.5161922574043274, -1.2196044921875, -0.7690654993057251, -0.6431620121002197, 0.5869811177253723, -1.5666389465332031, -0.3701243996620178, 0.7164204716682434, -0.5932841897010803, -0.6957408785820007, 0.3648330867290497, -0.4368952512741089, -1.2563852071762085, -1.48116135597229, 0.4108007550239563, 0.18253721296787262, 0.18870149552822113, 1.513218879699707, 1.5391079187393188, -0.8950491547584534, -0.9364557862281799, 0.02512911520898342, -0.27482354640960693, 1.0063294172286987, 1.0283586978912354, 0.10008707642555237, -0.29207056760787964, 2.5708515644073486, -0.6265033483505249, 0.0965602770447731, -2.9081127643585205, -0.00046309083700180054, 1.1887949705123901, -0.2376759648323059, -1.8798190355300903, 1.0511938333511353, 0.5778002142906189, 0.3477674424648285, 0.040590692311525345, -0.970464825630188, -0.0056758541613817215, 0.6447874307632446, 0.4737919569015503, -0.08509965240955353, -1.1519333124160767, -0.02007460594177246, -0.12365841120481491, -0.3784691393375397, -1.0059267282485962, -0.5744345188140869, -0.7264392971992493, 1.3936817646026611, 2.0902788639068604, 2.75373911857605, 0.979107141494751, -0.08520060777664185, 0.9670417308807373, 0.8167110085487366, -1.1228525638580322, -1.107703447341919, 0.16087034344673157, 1.0085901021957397, -0.9308015704154968, 0.4975719749927521, 1.251329779624939, -0.34669339656829834, 0.1160421222448349, -1.074323058128357, 0.4749472141265869, 1.0024018287658691, 0.6776801943778992, 2.4293153285980225, 0.11598697304725647, 0.27785125374794006, -0.4363885223865509, 0.2654440999031067, 0.42906755208969116, 0.5809255838394165, -0.28653207421302795, 2.501328229904175, -1.4096767902374268, 1.1885969638824463, 0.5052012205123901, -0.15126459300518036, -0.71364825963974, -1.3125250339508057, -1.5165023803710938, -1.5351459980010986, 0.10311181843280792, 0.12732911109924316, 0.31321418285369873, 0.775654137134552, 0.35976776480674744, -1.091263771057129, -0.17200946807861328, -0.4609011709690094, -0.07499561458826065, -1.104722023010254, 0.2781098484992981, 1.1163873672485352, -0.500443160533905, -0.8629677295684814, -0.5639746785163879, 0.38788822293281555, -1.1071001291275024, 0.6836204528808594, -1.2865978479385376, 0.6723421812057495, -0.7990485429763794, -1.3984086513519287, -0.9035809636116028, -1.094689130783081, -0.8461688756942749, -1.6307352781295776, -0.19037310779094696, -1.2474623918533325, -0.5661383867263794, 0.29612261056900024, -0.1200481727719307, -0.858918309211731, 1.257065773010254, -2.4606990814208984, -0.7379029989242554, -0.044243671000003815, 0.8243177533149719, -0.6360626816749573, 0.3088056743144989, -0.8005803823471069, 1.3141741752624512, 0.45112723112106323, 0.7173969745635986, -0.1660347282886505, 0.3571598529815674, -1.0198596715927124, 0.9974005818367004, -1.060341238975525, -1.1827726364135742, 0.8341718316078186, -0.24907217919826508, 0.25415149331092834, 1.4538136720657349, -0.4868169128894806, 1.683567762374878, 0.14177337288856506, -0.6348429918289185, -0.7790570855140686, -0.44332295656204224, 0.8232592940330505, -0.0785185918211937, -0.15197519958019257, 1.1588516235351562], [-0.06863895803689957, 0.001632882747799158, -0.5968341827392578, -0.701115071773529, -0.3250679671764374, 1.1111645698547363, 1.8095486164093018, -1.0729649066925049, -0.9682702422142029, -0.9250358939170837, -1.7029722929000854, -1.6040633916854858, -0.33634957671165466, 0.8275960087776184, -1.3371065855026245, 0.5880281925201416, 0.1268889158964157, 1.3329641819000244, 0.4774750769138336, 0.5730323791503906, -0.29886743426322937, 0.9152865409851074, 0.2885338068008423, -0.26968809962272644, 0.6291565895080566, 0.4206220805644989, 0.02524874359369278, 0.17599479854106903, -0.7775286436080933, 0.5904043912887573, -1.2251222133636475, -2.123885154724121, -0.4340483248233795, -0.8022807836532593, 0.5752789378166199, -0.029501119628548622, 0.15101508796215057, 1.3324129581451416, 0.3118063807487488, 0.2917226552963257, -1.2854101657867432, 1.4358201026916504, -0.34508568048477173, -1.1957836151123047, 0.18061771988868713, 0.5040631294250488, -1.3361806869506836, -0.19025202095508575, -0.8413637280464172, -0.2798418402671814, -0.7603729963302612, 0.5446217060089111, 0.00765722943469882, 0.13886785507202148, -0.016556520015001297, -0.36645781993865967, -0.6405830383300781, 1.001677393913269, 1.0976383686065674, 0.1597185730934143, 0.8820847868919373, 1.3756285905838013, -1.4066718816757202, -1.440370798110962, -0.5767371654510498, -0.8546183109283447, 0.012961892411112785, -0.8451526165008545, -0.9712399244308472, -0.07232695817947388, 0.23864123225212097, 0.3478997051715851, -0.020927658304572105, -0.7113611698150635, -0.6398971080780029, 0.48648399114608765, -1.3508518934249878, -0.6971444487571716, -0.23164761066436768, -1.6846404075622559, -0.8133692741394043, -0.6552579402923584, -0.2707612216472626, -0.31167274713516235, 0.9670817852020264, 0.06927327066659927, 0.020337652415037155, 0.6101984977722168, 0.35127824544906616, -0.12281874567270279, -0.6859895586967468, -0.037017133086919785, 0.9062168002128601, -0.3337458670139313, -1.1801331043243408, 0.02159923128783703, 0.14211143553256989, -0.7470263242721558, 0.2864986062049866, -2.794663667678833, 0.3086903989315033, -0.7262088060379028, 0.3340636193752289, -0.23809914290905, -0.06918999552726746, -1.256697654724121, 0.6334087252616882, -0.1698780506849289, 0.6352590918540955, 0.3706313967704773, -1.537522554397583, -0.5417840480804443, 0.8053475618362427, -0.5625858306884766, -0.32744523882865906, -1.7665706872940063, -0.44423359632492065, -1.2965006828308105, 0.08510419726371765, 0.2548231780529022, -0.36402982473373413, 0.6973710656166077, 0.04670725017786026, 2.354020118713379, -1.2370096445083618, 0.9294527173042297, 0.38181519508361816, -1.4156461954116821, 0.017994102090597153, -1.8703126907348633, -0.14430181682109833, -1.770725131034851, 0.029699040576815605, -0.340152770280838, 0.7035517692565918, 0.11057022958993912, 0.2676987648010254, 1.000833511352539, -2.3972182273864746, 0.49109727144241333, 0.9426792860031128, -0.09710103273391724, 0.7027665972709656, -1.257812261581421, 0.4219168424606323, -2.599562168121338, 0.1103823184967041, 0.5985225439071655, -1.3468199968338013, -0.7670586705207825, 0.0553409606218338, -0.5626999139785767, -0.3283708989620209, 1.152949571609497, -0.5150046944618225, -1.3294379711151123, 0.42208489775657654, -0.7838278412818909, -0.22977475821971893, -0.2926872670650482, -0.8320408463478088, 2.788498878479004, 0.5657051801681519, -0.517983078956604, -0.12447825074195862, 0.4166904389858246, -0.89084392786026, 0.1735396534204483, -0.8728792071342468, -1.4160475730895996, 2.0696728229522705, 0.14405833184719086, 2.7723805904388428, 1.8298269510269165, 1.8231300115585327, 0.20252978801727295, 0.43259158730506897, -1.532820463180542, 0.29392609000205994, 0.4567418694496155, -0.6942611336708069, -0.5467923879623413, 0.8069167733192444, -2.3851895332336426, -0.7730900049209595, 0.5439032316207886, -0.6533941626548767, -0.6525182723999023, 0.19735920429229736, 0.5975764989852905, -1.0237926244735718, -0.2521989643573761, 1.4124376773834229, -0.2777474522590637, -0.9086523056030273, -0.3208606243133545, 1.173845648765564, -1.1065672636032104, 0.17096851766109467, 1.7489993572235107, 0.1348525881767273, 0.6338534951210022, -0.3232420086860657, 0.36274194717407227, -0.41685017943382263, -0.037539590150117874, 0.8151989579200745, -0.5452284812927246, 1.050747275352478, 0.7453952431678772, 0.9234543442726135, -0.4371691346168518, 0.9950546026229858, 0.14895515143871307, -1.1674528121948242, 0.03831411153078079, 0.5436020493507385, 1.022767424583435, -1.0973641872406006, -1.8759175539016724, -1.3284263610839844, -2.058541774749756, 0.10944518446922302, 1.5029470920562744, 1.2084975242614746, 0.5623652935028076, 0.5827465653419495, 0.08079960197210312, -0.49842000007629395, 0.33443063497543335, 0.5763660669326782, -0.9568397402763367, 0.4935072064399719, 0.18586967885494232, 0.26213523745536804, -0.3211446702480316, 0.361918568611145, -1.3230655193328857, -0.48163536190986633, 0.18215851485729218, -1.2955402135849, -0.18105065822601318, -0.5866257548332214, -2.0804057121276855, -1.450189232826233, 0.38390690088272095, 0.5533922910690308, -1.6157077550888062, 0.639441192150116, 0.8025468587875366, -0.013911480084061623, -2.0513412952423096, -0.6978728771209717, -1.4977160692214966, 0.513297438621521, -1.6724121570587158, -1.14454984664917, 1.569060206413269, 0.14070846140384674, -0.17248877882957458, -1.2739380598068237, -1.0993000268936157, 1.0608599185943604, 0.5265676975250244, 0.30503740906715393, -0.24776332080364227, 0.1071871891617775, 0.49244457483291626, 0.9506697654724121, 1.544060468673706, -0.757343590259552, 1.4891937971115112, -0.4234577417373657, 1.8941857814788818, 1.2926437854766846, 2.098337411880493, 0.7590557336807251, -0.26311007142066956, 0.4535737931728363, 1.6783186197280884, -1.979451060295105, -0.4604058563709259, 0.4942828416824341, 0.979421854019165, 0.06628353148698807, 0.11980798095464706, -0.661291241645813, 0.35805046558380127, -1.1945604085922241, -0.9801020622253418, -0.09839682281017303, -1.7761825323104858, -1.2314715385437012, 0.7810314893722534, 0.1000351533293724, 0.16663116216659546, 0.6163548231124878, -0.341977596282959, -0.008745446801185608, 1.2565813064575195, -2.085808038711548, -1.3839508295059204, 0.19553589820861816, 1.0970863103866577, -0.2404351383447647, 0.4176800549030304, 0.8531749844551086, -0.05663331225514412, 0.43048372864723206, 0.5484919548034668, 0.3269149661064148, -1.263500452041626, -0.9044842720031738, 1.367463231086731, -0.0656566396355629, -0.42941534519195557, 2.29815673828125, 0.1942199021577835, 1.8259620666503906, -1.6723588705062866, 1.3562012910842896, 0.9726909399032593, -0.18171124160289764, -1.51844322681427, 0.233759343624115, 2.0274930000305176, -1.610816240310669, -0.6176183223724365, 0.7135424613952637, 0.24812909960746765, 0.3901316821575165, 0.2010994702577591, -0.5903711915016174, 1.201509714126587, 0.5922369360923767, -0.97947758436203, 1.0335053205490112, -0.19503137469291687, 0.09788800030946732, 1.2884007692337036, 1.1875560283660889, -1.2662990093231201, -1.1533653736114502, 0.6720912456512451, -2.1520838737487793, 0.505623459815979, 0.5593345165252686, 1.6039576530456543, 1.947372555732727, -0.8859760761260986, 0.03233997896313667, 1.0227528810501099, 1.492473840713501, 2.220341682434082, 1.0351588726043701, 0.15539349615573883, -1.380603551864624, -0.0792699083685875, -0.1121596246957779, -0.31471124291419983, 0.9442504048347473, 1.8500292301177979, 0.017643265426158905, 0.7393893599510193, 0.20346228778362274, 0.28445401787757874, -0.08761236071586609, 1.1003869771957397, 1.784894585609436, -0.08897940814495087, 1.1531903743743896, -0.009731084108352661, -1.1866363286972046, 1.0632989406585693, -0.7435556054115295, 1.121462106704712, -0.9940587878227234, -0.09381268918514252, 0.14829666912555695, -0.6871370673179626, 0.9233357310295105, 1.1816662549972534, -0.8295559287071228, 0.1329730749130249, 1.6681796312332153, -0.8551475405693054, 1.7583657503128052, 0.2717723548412323, 0.010792536661028862, -0.3451419472694397, -0.1010926365852356, 1.4154828786849976, -0.00015800073742866516, -0.2177727371454239, 0.9411606192588806, -0.010269543156027794, -0.5219583511352539, 0.10006968677043915, 1.012096643447876, -1.4518364667892456, -1.5713106393814087, 0.7147721648216248, 0.23220598697662354, -1.0370928049087524, -0.7624355554580688, 1.2070913314819336, 0.17521613836288452, 1.7389825582504272, -2.055063486099243, 1.3028100728988647, -0.6183930039405823, -0.7892963290214539, 0.27565255761146545, 1.9619860649108887, -3.168337345123291, -1.1650513410568237, -0.16341324150562286, 0.1137782484292984, -0.22917555272579193, 1.6593477725982666, 0.14583243429660797, 0.42606493830680847, 1.2140055894851685, -0.2638232409954071, -0.40320542454719543, -0.3329101502895355, 1.32504141330719, -0.863792359828949, 0.55806964635849, 0.4983266592025757, 0.26647016406059265, -1.0091865062713623, 1.2951085567474365, 0.6024467945098877, -0.7208073735237122, 0.6731545329093933, -0.12558579444885254, 0.39027732610702515, -0.5684536695480347, -0.27331510186195374, -0.20224058628082275, 2.3157248497009277, -0.926606297492981, 0.5554907321929932, -0.6955845952033997, 1.6059590578079224, 0.3827335238456726, 0.796707272529602, 0.9582217931747437, -0.2954384386539459, 1.8574745655059814, -0.7272899150848389, 0.47347474098205566, -0.5719390511512756, -0.32273563742637634, 1.0329012870788574, -1.371700644493103, -0.7868428826332092, 0.27663561701774597, 1.2982438802719116, -1.0103110074996948, -1.325728178024292, 0.3584209680557251, -0.996082067489624, -1.950440526008606, -0.9511076807975769, 1.575668454170227, 0.37614762783050537, 1.7597171068191528, -0.16053754091262817, -1.440148949623108, 1.5767589807510376, 0.02662176452577114, -0.5944956541061401, -0.9259021878242493, -0.8174115419387817, -0.5002688765525818, -1.1261168718338013, 0.28878456354141235, 2.0448157787323, -1.962572693824768, 0.7083531618118286, 0.03309343382716179, 1.2018336057662964, -1.327027440071106, -1.1979035139083862, 1.0761388540267944, -0.6687538623809814, -0.3932536244392395, -1.0073740482330322, 0.28806379437446594, -0.1539665311574936, 0.29017770290374756, -0.04012807831168175, 0.05210673436522484, 1.0314009189605713, 1.4294090270996094, -2.1527822017669678, 1.2384788990020752, 0.5844166278839111, -0.7429857850074768, -1.7605892419815063, 1.8036874532699585, -0.7383741140365601, -0.6679897904396057, 0.5709802508354187, -0.5313652753829956, -0.8148696422576904, -0.48506835103034973, 0.09754964709281921, 1.0174331665039062, 0.7146204710006714, 0.07659727334976196, 0.3052847981452942, -0.41511470079421997, 1.1049926280975342, -1.0427577495574951, -0.2834183871746063, -1.6572751998901367, 0.31472334265708923, -2.13260817527771, -1.3397165536880493, 0.4962216019630432, -0.580491304397583, -2.4576666355133057, 0.9298198223114014, -0.06985674798488617, 1.253212332725525, 0.13903526961803436, -0.8891725540161133, -0.8443660140037537, 1.1723417043685913, -0.35218867659568787, -0.49568411707878113, -2.1275084018707275, 1.3555734157562256, -0.5498533248901367, 1.1038399934768677, -0.8524460196495056, 1.9720757007598877, -1.1078327894210815, -0.3534625470638275, -2.4456586837768555, -0.5881473422050476, -0.6611748933792114, 0.7995308041572571, -0.7581138610839844, 1.3872288465499878, 1.301500916481018, -0.8183677792549133, -0.36502620577812195, -0.03296776860952377, -0.5738456845283508, 0.8735250234603882, -0.36341357231140137, 0.7757249474525452, 0.12359419465065002, 1.3470854759216309, 1.9461636543273926, 0.9825994968414307, 0.35125747323036194, -1.0020115375518799, 1.1270636320114136, 0.6362888813018799, 0.2188287079334259, 0.7560040354728699, -1.3775336742401123, 0.13988161087036133, 0.4868665039539337, 0.9854198694229126, 0.8016149997711182, 0.24041952192783356, 0.44482889771461487, -0.19756647944450378, -1.6481846570968628, 1.5194646120071411, 0.6432415246963501, -0.362806111574173, -0.2615309953689575, -1.1078693866729736, -0.18343685567378998, -1.313361644744873, -1.6332427263259888, -1.4892181158065796, 1.475538969039917, -1.1061450242996216, 0.4352242052555084, 0.721426248550415, 0.027193943038582802, -0.7067474126815796, -1.5746933221817017, 0.06688795238733292, 0.5405943989753723, 0.014232935383915901, -2.074901580810547, 0.7203919887542725, -0.5774538516998291, -0.08469317853450775, 0.5758594274520874, -2.215043544769287, 0.06875917315483093, -0.04487557336688042, 0.655440628528595, 0.5152878165245056, 0.03389977291226387, 0.23088400065898895, 0.5972955226898193, -0.04713577404618263, 0.5760185122489929, 1.6016724109649658, -0.572321891784668, 0.060525886714458466, -0.6453501582145691, 0.7123891711235046, -3.328890800476074, 0.712356448173523, 0.23506994545459747, 0.2012709379196167, -1.1954189538955688, -0.18399205803871155, 1.128807783126831, 0.6496633291244507, 0.6266745328903198, -0.6990429759025574, -0.029915165156126022, 0.04832364246249199, 0.7468677163124084, 0.6580910682678223, 1.7874016761779785, -1.4228096008300781, -0.35523825883865356, 0.13513131439685822, -0.7069681286811829, 0.13190309703350067, -0.39624857902526855, 0.28762930631637573, 1.0972354412078857, 0.17438557744026184, 1.7098332643508911, 0.1661035716533661, 0.2834654152393341, -0.7768709063529968, -0.3449344336986542, 0.1783304214477539, -0.22420838475227356, 0.3360101282596588, 0.9285911917686462, 1.250380277633667, -2.028416156768799, -0.19511273503303528, -1.7509227991104126, -3.112018346786499, -0.19455689191818237, 0.2005406767129898, 0.6334524750709534, -0.6790409088134766, -0.8040527701377869, -0.1876744031906128, 1.1243740320205688, 2.1342101097106934, -0.22189287841320038, -0.8966464996337891, 0.09571512788534164, -0.19257865846157074, -0.3736805021762848, -1.7370167970657349, 0.7855497598648071, 0.25997743010520935, -0.41627854108810425, -0.5344232320785522, 1.4981226921081543, 1.3686543703079224, -0.13357476890087128, -0.5467384457588196, -0.5235562920570374, 0.32074281573295593, -0.6718131303787231, 0.3142971694469452, -0.07176239788532257, 0.47373461723327637, -0.21550001204013824, 0.15272991359233856, 0.7338197827339172, 0.023252537474036217, -0.5006737112998962, -1.6178001165390015, -1.0136911869049072, 0.03209686651825905, 0.7464485168457031, 1.0600512027740479, -0.5975644588470459, 1.4888399839401245, -1.2082533836364746, -1.4912738800048828, 1.307542085647583, 1.5303481817245483, -0.43376296758651733, -1.6495089530944824, -0.5627599954605103, 0.6288275122642517, 0.7688460350036621, 0.876126766204834, 0.6970402002334595, 0.6723257303237915, 1.5152336359024048, -0.31021302938461304, 0.08474686741828918, 0.4707469046115875, 0.99503493309021, -1.6545238494873047, 0.7248133420944214, -1.1435259580612183, -0.27410024404525757, 0.45148494839668274, 0.10493333637714386, 2.431389570236206, 1.2638694047927856, 2.5262391567230225, -1.3726723194122314, 0.5683770775794983, 0.9778750538825989, -1.3128998279571533, 1.4124808311462402, -1.0421262979507446, 1.2102789878845215, 0.27538105845451355, 0.6930041313171387, 0.9968572854995728, 0.2097160965204239, -1.2697474956512451, -2.631155252456665, 0.7909628748893738, 0.35034093260765076, 0.9403708577156067, 0.1974024474620819, 0.4937908947467804, -0.08106952160596848, -0.4505915641784668, -0.9123635292053223, 0.04743029549717903, -1.0763914585113525, 0.010816268622875214, 0.47749534249305725, 0.5693138837814331, -1.8104790449142456, -1.605814814567566, -0.43417349457740784, -1.3038365840911865, 0.38599586486816406, -0.1144610345363617, -0.8482778072357178, -1.2999354600906372, 0.410072922706604, -1.5717819929122925, 0.6333848237991333, -0.33880823850631714, -0.9904175400733948, 0.33247408270835876, -0.12966422736644745, -0.8086315989494324, -0.08766254037618637, 0.34257885813713074, 1.5216286182403564, -0.2613057494163513, 1.0804054737091064, -0.9391229152679443, -1.3345080614089966, -0.04485944285988808, -0.8406051993370056, 1.2488584518432617, -0.14401693642139435, 0.5050835609436035, 0.5993229746818542, -1.54875648021698, 0.7102550864219666, -0.5557286739349365, 0.792813241481781, 0.38383716344833374, 0.65406733751297, 1.7542016506195068, -0.2770555913448334, 0.9968628287315369, -0.2584976851940155, 0.39792582392692566, -0.6627872586250305, 0.21392904222011566, 1.1620851755142212, -0.4892292618751526, -0.5535055994987488, -1.0439938306808472, 0.628483235836029, -0.5904549360275269, 0.18046726286411285, -0.6883552670478821, -1.719446063041687, 0.16538305580615997, -0.07997901737689972, -0.8841456770896912, -0.8500804901123047, -0.9445966482162476, -1.322393774986267, -1.0480761528015137, 0.5611815452575684, 1.7096672058105469, -1.4694364070892334, 0.5569010972976685, 0.1634024679660797, 1.2025399208068848, 0.5068564414978027, -0.3660271465778351, 1.139599084854126, 0.1451798528432846, 0.350521981716156, -0.23412759602069855, -1.4191913604736328, 1.2551449537277222, -2.421058416366577, 1.138763189315796, -0.651953935623169, 2.3982291221618652, -0.5138033032417297, 0.2691391110420227, 0.03537644445896149, -0.9306283593177795, -2.1634693145751953, 0.8524991273880005, 0.33316272497177124, -1.178234577178955, 1.2033436298370361, 0.21342647075653076, 0.29570144414901733, 0.5947298407554626, -0.172620490193367, 0.4453527629375458, 0.3282511234283447, -0.3292384743690491, 1.2388877868652344, -0.865990400314331, -0.08286864310503006, -1.4895776510238647, -0.8571877479553223, -0.1070130467414856, 1.0370253324508667, -1.1345171928405762, 0.6733874678611755, -0.8488099575042725, 1.444566249847412, 0.168406143784523, -1.151554822921753, 0.3945153057575226, 2.587082862854004, 0.12120361626148224, 2.65141224861145, 0.13404805958271027, 2.439535140991211, 0.5756285190582275, -0.05322394520044327, -0.004852729849517345, -0.5667971968650818, 0.23041784763336182, -0.9204588532447815, -1.8328086137771606, -1.3396316766738892, -0.05259839445352554, 0.9727972149848938, 0.10322410613298416, -0.03985985741019249, 0.2286890596151352, 1.0511430501937866, 0.48385071754455566, 1.4893416166305542, 0.4406895637512207, 0.04811519756913185, -1.2132753133773804, -0.5627217292785645, 0.5310636162757874, -1.3865123987197876, -1.4984145164489746, -2.1767046451568604, -0.14360491931438446, 0.9061111211776733, 0.22334465384483337, -0.5141093134880066, -0.18849585950374603, 0.08931785821914673, -1.033583402633667, 0.3620247542858124, 0.22921881079673767, 0.3924788534641266, 0.49517399072647095, -0.8066008687019348, 0.8234019875526428, -0.39034077525138855, 0.6182987689971924, -0.1234469935297966, -0.6310148239135742, -1.253779411315918, -0.04798293858766556, -0.9540029168128967, 1.9499328136444092, 1.4802191257476807, -1.336835503578186, 0.278004914522171, 0.09599663317203522, 2.0297858715057373, -1.0267386436462402, -1.098423957824707, -0.8048713207244873, -0.7626933455467224, 0.8537608981132507, 0.8544619679450989, 0.08143241703510284, 1.4568281173706055, 0.41583091020584106, -0.8999215364456177, -0.4417232871055603, 1.8740625381469727, -0.0768827572464943, -1.1894235610961914, -0.7736257314682007, -1.0905513763427734, 3.090070962905884, -0.433102548122406, -0.8893294930458069, 0.28410783410072327, 1.1761292219161987, -0.8614556789398193, 0.21863733232021332, -0.09398037940263748, 0.8675580024719238, -1.0792200565338135, 0.13727639615535736, -0.5486891269683838, 1.5793249607086182, -1.2659807205200195, 0.32614171504974365, -0.1217212975025177, -0.3993852436542511, 1.4060848951339722, -0.9478921890258789, -0.17898599803447723, 1.4416589736938477, 1.5866749286651611, -0.806862473487854, 0.6029189229011536, -0.36333218216896057, 0.19755591452121735, 1.4050222635269165, 0.07036714255809784, -0.49547189474105835, -0.3186243772506714, -0.7923300862312317, -0.8728421926498413, 0.2172316163778305, -0.7022545337677002, 0.41195595264434814, 0.4272059202194214, -0.22130118310451508, -1.508509635925293, -0.7402024865150452, -0.38733625411987305, -0.6576058268547058, 0.4601718783378601, 1.151841163635254, 1.517653465270996, -0.3313484787940979, -0.6385206580162048, 0.3852730393409729, -0.19328947365283966, 1.3695273399353027, 0.9078420996665955, -1.1207897663116455, 0.41622987389564514, 0.11647194623947144, 0.13713526725769043, 1.1265863180160522, -1.0210883617401123, 1.5531469583511353, 0.4206492602825165, 1.3965904712677002, -1.1124505996704102, -0.38644084334373474, 1.0401896238327026, -0.38303670287132263, -0.5510305166244507, 0.12452942132949829, -2.083847999572754, 1.2522975206375122, -0.4483630359172821, 1.0494405031204224, -0.43922513723373413, -0.36120977997779846, -0.6715235710144043, -0.06755780428647995, -0.7073387503623962, 0.2555716633796692, 0.6324165463447571, 0.4829625189304352, -1.0252543687820435, -1.3225959539413452, 0.8174314498901367, -0.9397149085998535, -0.3358626663684845, -0.17652001976966858, 0.4923442602157593, -0.33052921295166016, 0.7153288125991821, -1.2793817520141602, -0.054016418755054474, -0.14806054532527924, -1.4858719110488892, 0.07771629840135574, -1.3577593564987183, -1.1077449321746826, -0.44261491298675537, 0.1229814738035202, 0.3120589554309845, -1.7608460187911987, -1.488911509513855, -0.11682257056236267, -0.7736832499504089, 0.5424062609672546, -1.2467436790466309, 1.1632561683654785], [0.16245046257972717, -1.5997859239578247, 1.4784917831420898, 0.2442212551832199, -1.625170350074768, -0.0018402658170089126, -0.8115169405937195, -0.030331062152981758, 0.12605182826519012, -1.6476013660430908, -1.5231877565383911, 0.35206660628318787, -0.006727619096636772, -0.8188698887825012, -0.17992541193962097, 0.07690311223268509, 0.8812123537063599, -0.7038828730583191, -0.2860136330127716, 0.44586360454559326, -0.34985318779945374, -2.580487012863159, -0.2524745762348175, -0.037942126393318176, -1.0046559572219849, 0.036520637571811676, 2.606642723083496, -1.797749638557434, 1.0779932737350464, 0.30981531739234924, -1.8022209405899048, -0.04077731817960739, 0.6323044300079346, 1.3343756198883057, -1.124598741531372, 1.7200028896331787, -0.23469026386737823, 0.3853660225868225, 0.8687447309494019, -0.9066377282142639, 0.13564516603946686, -0.1834115982055664, -0.24737970530986786, 0.07282088696956635, 0.6166179776191711, 1.0162713527679443, -0.07744178920984268, 0.5144230723381042, 1.3601624965667725, 0.75666743516922, -0.5303341150283813, -0.5670514702796936, -0.5201851725578308, 0.729663074016571, 0.5192928314208984, -0.3964247703552246, 1.0959014892578125, 0.42800310254096985, -1.5267770290374756, -0.24043698608875275, -0.20146450400352478, 0.4525301158428192, 1.604306697845459, -1.0639357566833496, -1.7714293003082275, 0.6510879397392273, -0.5020098686218262, -0.7580441236495972, -1.5780068635940552, -2.695448875427246, 0.3469996154308319, -0.1312020719051361, -0.7185274362564087, -0.04524116963148117, 0.9037639498710632, -2.6113979816436768, 0.774848222732544, 1.0048562288284302, 1.1401876211166382, -0.6423415541648865, -2.3893206119537354, -0.9210942387580872, 1.2003602981567383, -2.0833945274353027, -0.3759666979312897, 1.044826865196228, 0.6721044778823853, 0.3445127308368683, 0.2319168895483017, -2.0782642364501953, 0.5116704106330872, -0.007352963089942932, -1.146305799484253, -1.7888484001159668, 0.9844740033149719, 0.4118626117706299, 0.23541593551635742, 0.4013988971710205, 0.8737375736236572, 0.5588361024856567, -0.10921533405780792, -0.7917513251304626, 0.8902595043182373, 0.25110024213790894, 1.4256621599197388, 1.192273497581482, 2.627556800842285, -0.48653313517570496, -0.6853069067001343, 2.302284002304077, -0.5783618688583374, -0.8868829011917114, -1.0704901218414307, 0.29685619473457336, 0.7586002349853516, 0.9900512099266052, -0.7861055731773376, 1.3025826215744019, -1.7191823720932007, 0.1782316416501999, 0.4555530846118927, -2.5790140628814697, 0.5618975758552551, -0.04294019937515259, -0.894295871257782, -1.2046308517456055, -0.8583170771598816, 0.24590468406677246, -0.43043455481529236, 0.40864095091819763, -0.2793055772781372, 2.287909984588623, 2.243586540222168, -2.250782012939453, 0.48805293440818787, -1.698689341545105, 0.016026198863983154, 1.4232656955718994, -1.044897437095642, -0.2760012447834015, 0.34912142157554626, 0.7196975350379944, -1.441719889640808, 0.5476716756820679, -1.7768830060958862, 0.7600132822990417, -0.2555626630783081, -0.3444858193397522, -0.7581606507301331, 0.32760870456695557, -0.5323525071144104, 1.1685495376586914, 0.4936072826385498, -1.0317409038543701, -0.46500715613365173, -2.2813353538513184, -1.3967053890228271, -1.119492769241333, -2.1519975662231445, 0.4375731348991394, 0.24405783414840698, 1.0234590768814087, 0.32743149995803833, 1.178094744682312, -0.9426558017730713, -1.695857286453247, -1.0698113441467285, -1.558939814567566, -0.007529341150075197, -1.3565632104873657, -0.6592150330543518, -0.40297800302505493, -0.4394518733024597, -1.1506695747375488, -0.9312676191329956, 0.45339852571487427, -0.44272127747535706, 0.9006852507591248, -0.7873431444168091, -0.2533891797065735, 0.24701456725597382, -0.5076060891151428, -1.4397778511047363, 0.20696623623371124, 1.1528178453445435, 1.0817779302597046, -1.257506012916565, -1.141872525215149, -0.4625732898712158, 0.06406503915786743, 0.3684958517551422, 0.7580034732818604, 0.5944104790687561, -1.2470356225967407, 2.3204591274261475, 0.1809026598930359, -0.8913058638572693, 0.5828869342803955, -1.1038652658462524, -1.0308157205581665, -1.7443487644195557, 0.10294970124959946, 0.15415726602077484, 1.0537567138671875, -1.081421136856079, -1.6325732469558716, 1.0440220832824707, -0.016104264184832573, 1.7738054990768433, -0.9837877750396729, 0.6694644689559937, -0.40253254771232605, -1.4203522205352783, -1.87840735912323, -0.04374060034751892, 0.6620215177536011, -0.5519059896469116, 0.7328453660011292, 0.12152490019798279, -1.6459355354309082, 1.546584963798523, -0.8768583536148071, 0.5291771292686462, -1.2077702283859253, 0.5745340585708618, 0.826090395450592, -0.293953001499176, 1.1866261959075928, 0.17831598222255707, 0.9462872743606567, -1.602213740348816, 1.4509490728378296, -0.7184996604919434, -1.154076099395752, 0.6836126446723938, -0.34128227829933167, 0.02343609370291233, -0.2644658386707306, -0.08039136230945587, 0.9007168412208557, 0.009606977924704552, 0.5050386190414429, -0.96784508228302, -0.6674244403839111, 1.5166165828704834, 2.320660352706909, -0.12085682153701782, 0.2114027738571167, 1.136942982673645, 1.5297902822494507, 1.144471526145935, 0.10667017847299576, -1.2533466815948486, 1.1734427213668823, 1.1619873046875, 0.3949822783470154, -0.7148540019989014, 0.7569019198417664, -0.7060108184814453, 1.2546460628509521, -1.7903780937194824, -0.4697059392929077, -0.3283720910549164, 1.2425727844238281, 2.256201982498169, -0.4988205134868622, 1.3220274448394775, -1.3378684520721436, -0.5742658376693726, -0.5140061974525452, 2.2633914947509766, 0.37712594866752625, -0.6041563749313354, 0.5507131218910217, 0.003988149110227823, 0.2594587206840515, -0.5397092700004578, -0.63371741771698, 2.5365960597991943, -1.2590011358261108, -0.19952428340911865, -0.44146499037742615, 0.13335366547107697, 0.5299568176269531, -0.39982369542121887, 0.30312860012054443, -1.6664987802505493, -1.4402519464492798, -1.3865832090377808, -0.3530440032482147, 1.89380943775177, -0.4655391275882721, -0.9752053022384644, 1.5923093557357788, -0.3495151102542877, -0.5330906510353088, -0.6331273913383484, -0.8402194380760193, 0.8030074238777161, -0.7073106169700623, -1.6188031435012817, 0.8600112795829773, -0.8075100779533386, 1.4615092277526855, 0.0005271336412988603, 1.0468802452087402, 1.0357716083526611, -1.0420950651168823, 1.0050684213638306, 0.739597499370575, -1.1560927629470825, 0.34811675548553467, 0.08096495270729065, -1.5388808250427246, -0.9539022445678711, -1.5536645650863647, 0.40375539660453796, 0.07560642808675766, 0.39932793378829956, 1.4469499588012695, 1.345915675163269, 0.1270194798707962, -0.5078305602073669, -0.9475294351577759, -0.8500521779060364, -0.3338969051837921, 0.6377760767936707, -0.6443952322006226, 0.6150326132774353, 0.9146843552589417, -1.0330212116241455, 0.624484658241272, -0.010563475079834461, 0.7055137157440186, -0.05187492445111275, 0.18953436613082886, -0.6957015991210938, -0.36328014731407166, -0.9178165793418884, 1.1370846033096313, 1.5675944089889526, -2.802661180496216, 0.7368305921554565, -0.1713479459285736, -0.3793109357357025, 0.32523244619369507, -0.6330361366271973, 0.748869001865387, -0.13977433741092682, -0.2834366261959076, -0.6043285727500916, 0.4871264398097992, 0.4617866277694702, 0.3653962314128876, -1.2008967399597168, -0.5773998498916626, -0.7014292478561401, -0.11832579970359802, 0.15656088292598724, 0.7331355214118958, -0.13722822070121765, -0.6151620745658875, 0.6223028898239136, -0.5759983658790588, 0.540791392326355, 0.20360229909420013, -0.689397394657135, -0.2382449209690094, -0.11254356801509857, 1.5642071962356567, 1.7794102430343628, 0.5312470197677612, -0.28775590658187866, 0.42684969305992126, -0.9731307029724121, -1.7589067220687866, 0.22465087473392487, 0.5428448915481567, -0.8345850706100464, -1.6096577644348145, -1.4947274923324585, 0.7515667676925659, -0.4736001193523407, -1.347583532333374, -0.6890740990638733, -1.7483071088790894, -0.02107602171599865, -1.1583764553070068, -0.9512814283370972, 0.8989070653915405, -0.1464548408985138, 1.249079942703247, 0.4090602993965149, -0.37268927693367004, 1.1954724788665771, 0.28873559832572937, -0.8851286172866821, 0.1734340637922287, 0.6386039853096008, -0.7387723326683044, 0.8130145072937012, 1.0355802774429321, 0.6570202708244324, -0.9250519871711731, -0.16313597559928894, -0.14941184222698212, -0.20415814220905304, 1.043684720993042, 1.1546629667282104, -0.3132612407207489, 0.40896180272102356, 0.726155161857605, -1.302863597869873, 0.1689528077840805, 1.0557582378387451, -0.5391804575920105, 1.4356857538223267, -0.41770440340042114, -0.1970425844192505, -0.15130001306533813, 0.64895099401474, -0.7921930551528931, -0.303842693567276, 0.40612515807151794, 0.7002919912338257, 0.3216905891895294, -0.5876120924949646, -1.766411304473877, -0.5219591856002808, 0.31532952189445496, 0.44129499793052673, 1.629180669784546, -1.860345721244812, -0.11854495853185654, -0.22548100352287292, -0.6666353344917297, 1.7207648754119873, 0.886733889579773, -0.004294454585760832, 2.398397207260132, 0.5731207132339478, 0.610643208026886, 1.1538605690002441, -1.0276587009429932, 2.9054815769195557, 0.2424655705690384, -1.262795329093933, 0.056451454758644104, -1.048667550086975, 1.633249282836914, 0.48287394642829895, 0.3014155328273773, 0.6037920713424683, 0.16564622521400452, 0.5882319211959839, 1.349434733390808, 1.3732346296310425, 0.7404552102088928, -0.12263020873069763, 1.45232355594635, 0.028199603781104088, 1.4753459692001343, 0.494024395942688, 0.2350205034017563, 0.9968190789222717, 0.25419941544532776, 0.3471348285675049, -0.47904592752456665, -0.7822391390800476, 0.07002053409814835, 0.9469677209854126, 0.11399038136005402, 0.9922058582305908, -0.8908786773681641, -0.6960545778274536, 0.9581799507141113, 0.011635098606348038, -1.2678825855255127, -0.3378984034061432, 0.2617044746875763, 2.38956356048584, 0.5511108040809631, -0.02596469223499298, 0.32865914702415466, 1.7461261749267578, -0.6616421937942505, -0.09672748297452927, 0.6363207101821899, 0.810032844543457, -1.4362047910690308, 1.113478660583496, -0.760861873626709, 0.0605539046227932, 0.7875242829322815, -0.012489969842135906, 0.09152224659919739, 1.0362733602523804, -1.231959581375122, -0.2615363299846649, 1.9463034868240356, -0.4318505823612213, 0.6764607429504395, -1.1274478435516357, 0.41218769550323486, 0.17265813052654266, -0.3995633125305176, 1.0109925270080566, 0.09644800424575806, 0.48618918657302856, 1.5974388122558594, -1.3475412130355835, 0.2552618980407715, -1.1851069927215576, 1.456646203994751, -0.10484404116868973, -0.4163196086883545, 0.2308468520641327, 0.3357928395271301, 1.7463572025299072, -0.0037475530989468098, 0.26667773723602295, -0.38865596055984497, 0.7285867929458618, 0.8133199214935303, 0.5891467332839966, -0.10949745029211044, 0.43581074476242065, 1.2604410648345947, 0.7921884655952454, -0.3226163685321808, -2.24684476852417, -0.26585260033607483, -0.3227900564670563, -1.586463212966919, 0.024379055947065353, -0.966241717338562, 1.8110862970352173, -1.4754910469055176, -1.3984330892562866, -0.06684363633394241, 0.06414567679166794, 0.38637328147888184, 0.24669472873210907, 0.8177298903465271, -0.28585171699523926, -1.071670413017273, -0.6210607290267944, 0.16369125247001648, 1.7346216440200806, -0.08866626024246216, 0.00023899416555650532, 0.4418553113937378, -1.5590102672576904, 1.317261815071106, -1.1479086875915527, 1.7676626443862915, 0.8430667519569397, 0.21711419522762299, -0.3160938322544098, 0.017776252701878548, -0.04363029822707176, 0.38656333088874817, 1.137894630432129, 0.3762778341770172, -0.670048177242279, -0.10996948182582855, -0.5793690085411072, 2.7447469234466553, 1.1107749938964844, 0.06519350409507751, 1.119499921798706, -1.500160574913025, -1.4423490762710571, -1.2700005769729614, 0.8995966911315918, -0.6416310667991638, -1.00685715675354, -1.6597625017166138, 0.722115159034729, 1.4043368101119995, -0.128914937376976, -0.43567603826522827, 0.9022433161735535, 0.11492919921875, 1.0629695653915405, 1.1684757471084595, 0.017902664840221405, -0.728458046913147, -0.667853593826294, -1.2458090782165527, -0.8282570242881775, 0.7127615809440613, 0.01825808919966221, 1.443813443183899, 2.0511157512664795, 0.7422664165496826, 0.23804688453674316, 0.9170637726783752, -0.7304422855377197, 0.3321370482444763, -0.8487751483917236, 0.5778800249099731, -1.130501389503479, 0.17600320279598236, 3.141209363937378, 0.16023296117782593, 0.6746737957000732, -0.46128562092781067, 0.6481190323829651, -1.2467130422592163, 0.30356231331825256, 2.130833625793457, 0.49973323941230774, 0.41863662004470825, -0.3598313331604004, -1.0720261335372925, 0.6107902526855469, -0.34581586718559265, 0.15077528357505798, 0.623993992805481, 0.0866522267460823, 0.5407535433769226, -0.9092578887939453, 0.5347980856895447, 0.1319265067577362, -1.013630747795105, 0.014381240122020245, -0.5384984612464905, 0.7020558714866638, 0.47007080912590027, 1.0759496688842773, -0.26388368010520935, 0.3667638599872589, -0.7063093185424805, -0.9363008141517639, -0.9802168607711792, -0.07402277737855911, -0.8339616060256958, 0.06524922698736191, 0.8099350333213806, -0.9059073328971863, 0.7703547477722168, -0.35087427496910095, -0.790473461151123, 0.07841303944587708, -0.12922652065753937, -1.1446187496185303, 0.3432200849056244, 0.47192662954330444, 0.34247419238090515, 2.7549407482147217, -0.131236270070076, -1.7649301290512085, 1.0953466892242432, -0.6132984161376953, 0.2337239682674408, -0.259892076253891, 1.2639145851135254, 0.6575609445571899, -0.22978739440441132, -0.3888859152793884, 1.8377772569656372, -0.8587746024131775, -0.22597891092300415, -0.37134093046188354, 0.951526939868927, -2.107942581176758, -1.5627813339233398, 0.2956148087978363, 0.003922367002815008, 1.2300779819488525, -2.588792562484741, -0.2622479498386383, -1.0015860795974731, -0.6970757246017456, 0.9590238928794861, -0.8442811369895935, 0.5573127269744873, 0.12028543651103973, 0.6093994379043579, 0.33102384209632874, -0.2525556981563568, -0.6679083108901978, -1.1086101531982422, 1.5558029413223267, -1.331782341003418, 1.1918147802352905, 0.7359306216239929, 0.5815721750259399, -0.7943251132965088, 0.08723848313093185, 1.998383641242981, 0.1513817310333252, 1.496910572052002, -0.26317673921585083, -0.418234258890152, -0.9126753807067871, 1.5740867853164673, 0.6664392948150635, 0.14252500236034393, 1.443859338760376, -0.813088595867157, -0.908553421497345, 0.005470555741339922, -0.6917070746421814, -0.6947162747383118, 0.3744162321090698, -1.0145399570465088, 0.18245398998260498, 1.1776137351989746, 0.7934877872467041, -1.0206634998321533, 1.546036958694458, -1.6166976690292358, 0.3854302763938904, 0.7114783525466919, 0.2860296070575714, 0.003281754208728671, -0.8475073575973511, -1.1801759004592896, 0.2456180304288864, 0.012129590846598148, -0.7017015814781189, -0.6950321793556213, -0.3857491612434387, -1.87613844871521, -1.0017155408859253, -0.5958606004714966, -1.8530057668685913, -0.5887014269828796, 1.0824096202850342, -0.9399545192718506, -0.8548591732978821, -0.39622148871421814, 1.098408818244934, 0.15622290968894958, 0.2672556936740875, -0.285795658826828, -0.8699045777320862, 0.5813184976577759, -1.3894250392913818, 1.9976346492767334, -0.49996110796928406, 0.5898687243461609, -0.2365178018808365, 0.19284574687480927, 0.2631000876426697, 0.9599603414535522, 1.5548114776611328, -1.1888879537582397, 1.2545340061187744, 0.038972266018390656, 0.9813696146011353, -0.18259769678115845, -1.0724422931671143, 0.5161995887756348, -0.02258894592523575, -1.2898629903793335, 0.35757142305374146, -3.1751575469970703, -0.1394060254096985, -1.7986841201782227, -0.1622583568096161, 0.9414415955543518, -0.4968481659889221, -1.6466526985168457, -0.5225300788879395, 1.184544324874878, 0.6072099804878235, 0.46285122632980347, 0.38847076892852783, 0.9480898380279541, 0.7470886707305908, 1.144313097000122, -0.46026238799095154, -0.5953424572944641, -0.12445960938930511, -0.4357192814350128, -1.240842342376709, -0.5492792129516602, 1.20261549949646, -1.6194297075271606, -1.5150198936462402, 1.1170437335968018, 0.16313114762306213, 0.00791601650416851, -0.39807364344596863, 0.5627472996711731, -0.04281037300825119, -0.49804311990737915, 0.35819461941719055, -0.268932580947876, -0.38164231181144714, -0.9021884202957153, 0.6006737351417542, 0.11829665303230286, 0.548438310623169, 0.13140740990638733, -2.0632851123809814, 0.6763169765472412, -0.7604605555534363, 1.4478703737258911, -0.025261634960770607, 0.3850211501121521, -0.590584933757782, 1.621846079826355, 0.8400936126708984, 1.4122200012207031, -0.2314474880695343, 1.0746465921401978, -0.5944321155548096, 2.8418939113616943, 0.8221408724784851, -1.035356044769287, -0.8050032258033752, 0.7964022159576416, -2.0547609329223633, -1.0071954727172852, -0.8588574528694153, 0.3134744465351105, 0.055290576070547104, -0.48419132828712463, 1.1582196950912476, 0.7051624059677124, -0.07655247300863266, -0.31226566433906555, -0.4910331964492798, -0.5033772587776184, 0.06961403787136078, 0.5695428252220154, 1.337965488433838, 0.5754507780075073, -1.5948610305786133, 0.9690554141998291, -0.422867089509964, -1.5721911191940308, -0.8572430610656738, -0.3952988386154175, 1.6284385919570923, 0.4900122880935669, 1.2028720378875732, -0.8367791771888733, -1.0243785381317139, -0.7651628255844116, -0.7301703095436096, 0.14506743848323822, -0.9637966752052307, -0.4643540680408478, 0.8330260515213013, 0.22445684671401978, 0.9263679385185242, -0.1906481832265854, 0.3392251133918762, -0.7500208616256714, -0.5043633580207825, 0.9299467206001282, 0.718866765499115, -1.924057960510254, 0.629770040512085, 1.250685691833496, 0.5233644247055054, 1.3640559911727905, 0.5168212056159973, 0.41621634364128113, 1.2992969751358032, -1.8558368682861328, -1.9761550426483154, -0.606987476348877, -1.0023274421691895, -2.468583345413208, 0.2788701057434082, -1.419735312461853, -1.0704421997070312, -0.10749725252389908, -1.471289873123169, -0.0713229700922966, -1.094639778137207, 0.6140056848526001, -0.2745119631290436, 0.014280249364674091, -0.9392461180686951, -1.9426125288009644, -0.30553510785102844, 0.96797776222229, 1.80413818359375, -1.2411830425262451, -0.14418554306030273, 2.204124927520752, -0.7233940958976746, -0.8297209143638611, 0.44430920481681824, 0.007181344088166952, 0.8216026425361633, 0.4500923454761505, 0.049722861498594284, 0.6496325731277466, 0.3250930905342102, -0.21093019843101501, -0.2897905707359314, -0.9836767315864563, 1.0529319047927856, 2.803518056869507, 0.2518240809440613, 0.09818015992641449, -0.9908472895622253, 0.17110298573970795, -1.3693491220474243, -0.07114948332309723, -0.6306103467941284, -1.8873820304870605, -1.7667244672775269, -0.4578331708908081, -1.4386584758758545, 1.0578913688659668, 1.4733093976974487, 2.2611374855041504, 0.1330798864364624, -1.2887481451034546, 0.3762493133544922, -1.1473333835601807, 0.40272852778434753, -0.4349316954612732, -1.3171615600585938, 0.5834476947784424, 2.067023754119873, 0.8519027829170227, 0.007252094801515341, -1.380661964416504, -1.2742269039154053, -0.5410599112510681, 0.7535938024520874, -1.647072196006775, -1.4095650911331177, -0.7670910358428955, -0.3256436884403229, -0.8017517328262329, -2.09444522857666, -0.9649673700332642, -0.16217730939388275, -1.0022075176239014, -1.1014715433120728, -0.9589925408363342, -0.33826956152915955, 0.6710600852966309, 0.41736263036727905, -0.42915746569633484, -0.7656196355819702, 0.18354079127311707, -1.0919803380966187, -1.798660397529602, -1.2212400436401367, -1.4431555271148682, -0.9153724312782288, 0.8102695345878601, -0.9167050719261169, 0.6094037890434265, 2.3774068355560303, 0.34324008226394653, -0.4654345214366913, -1.8201717138290405, -0.582146406173706, -0.7792198657989502, 0.4627806842327118, -2.2167601585388184, 0.7328311204910278, -0.19993987679481506, -0.2604491114616394, 1.6476328372955322, 0.3744586110115051, -0.8657459020614624, -0.29841628670692444, -0.7952526211738586, 0.7798953056335449, -1.3486357927322388, 0.4598131775856018, 0.08679072558879852, 2.6121561527252197, 1.5361390113830566, -0.18513642251491547, -1.1904691457748413, 0.7469556331634521, 0.5006769299507141, -0.5882701873779297, 2.306607246398926, 0.11072400212287903, 1.7552496194839478, -0.6516909003257751, -0.4227529764175415, -1.1196376085281372, -0.5013113021850586, 1.3624929189682007, -0.616307258605957, -0.40436050295829773, 1.3656316995620728, -0.5693819522857666, -0.3406088948249817, 0.6478807926177979, -1.6967154741287231, -1.1929497718811035, -0.484993040561676, 0.5555369257926941, 0.5832334756851196, 1.3105199337005615, 1.102501630783081, -0.4101722538471222, 0.03839180991053581, -0.48082977533340454, -0.32612496614456177, -0.05727236717939377, -0.19561730325222015, 1.431885004043579, -1.3978447914123535, -1.8847781419754028, 1.1382179260253906, -0.06380034238100052, 1.4685208797454834, -0.526824414730072, -0.2604351043701172, 0.05705232173204422, 0.27370017766952515, 0.6528955101966858, -0.16373203694820404, -0.7590435743331909, -1.043311595916748], [-1.3852128982543945, -0.5539163947105408, -0.1583002507686615, -1.6522806882858276, 0.1716078519821167, 1.409576177597046, 0.33053886890411377, -0.2230820208787918, -0.8840112090110779, -0.4044409692287445, -1.1622055768966675, 1.5205819606781006, -0.6168389916419983, -0.520851731300354, 0.7463388442993164, -0.8191384077072144, -0.11666316539049149, -0.4386084973812103, -0.5658903121948242, 0.006850098725408316, -0.8498218655586243, 1.0693687200546265, 0.07845112681388855, 0.13924343883991241, 0.3076256513595581, -0.9109553694725037, 0.1639200747013092, 0.09438818693161011, 1.3299204111099243, 1.2303043603897095, 0.281605988740921, -0.12675070762634277, 1.6397572755813599, 0.03375251591205597, 0.055775854736566544, 0.3056347966194153, -1.2611234188079834, 1.0350240468978882, -0.585502028465271, -0.5219393968582153, 1.2169826030731201, 0.28320077061653137, 0.40152689814567566, -0.7334464192390442, 1.510352611541748, 1.7501736879348755, 1.6563098430633545, -0.056685082614421844, 0.3081342875957489, -0.15157295763492584, -0.7016610503196716, -0.7966153025627136, 1.562564492225647, -1.250749945640564, 0.6306648254394531, 1.0449261665344238, 0.7984471917152405, 2.074486494064331, -2.1484622955322266, -0.1802888959646225, 0.5606463551521301, -0.7630429267883301, 0.3754827082157135, -0.9909757375717163, 0.42445430159568787, -0.4618329107761383, -0.8889340758323669, -0.520819902420044, -0.8716334700584412, -0.5698298811912537, -1.2997570037841797, -0.1615171730518341, 1.0683283805847168, -0.0659283995628357, 0.2583516836166382, -0.027122484520077705, 2.02860426902771, -0.42402124404907227, 0.5223158597946167, 0.2874663770198822, -0.2747388184070587, -0.4734030067920685, -0.1620577722787857, -0.703665018081665, 0.7938681244850159, 0.2398659735918045, 0.49500301480293274, -0.09855898469686508, -0.521236002445221, -2.6958494186401367, 0.7456067800521851, -1.3254567384719849, 0.33177444338798523, 0.26360976696014404, 0.06614608317613602, -0.6912276744842529, 0.05532762408256531, 0.6049550771713257, 0.34797918796539307, 0.07540050894021988, -1.253403902053833, 1.482775092124939, -0.8456467390060425, 1.8129023313522339, -0.2675732970237732, -0.9664640426635742, -0.28956034779548645, 0.34971731901168823, -0.5882545113563538, 0.2565280795097351, 0.040446650236845016, -0.5089619159698486, -0.3718253970146179, 0.7243191599845886, 0.9093146920204163, 1.4608979225158691, 1.6794651746749878, 0.41307541728019714, 1.397965669631958, 1.1443848609924316, 0.805119514465332, -0.17481708526611328, 1.457014799118042, -2.0583574771881104, -0.287796288728714, -1.1068397760391235, -0.36796629428863525, 0.2926599681377411, -0.5330719351768494, 0.07942421734333038, 0.628494918346405, -1.3547053337097168, 0.16051064431667328, 1.3112387657165527, 0.017636533826589584, 1.4131354093551636, 0.18669015169143677, 1.660961389541626, -0.34894394874572754, 1.3983973264694214, 0.32505133748054504, -1.1562358140945435, 0.3669779300689697, 2.9236299991607666, -1.1386390924453735, 0.6764446496963501, 0.280694842338562, 0.531688392162323, 1.9914336204528809, 2.0981318950653076, 0.4159710109233856, 0.7198091149330139, -0.38261905312538147, -0.41193899512290955, 0.47794899344444275, 0.471943736076355, -1.7350918054580688, -0.6883476376533508, 0.6338327527046204, -0.3904079496860504, 1.5925925970077515, -0.7451104521751404, 0.6581289768218994, 1.7676538228988647, 0.40996330976486206, -0.5010040402412415, -1.5402036905288696, -0.558088481426239, -0.8177971839904785, -1.5823087692260742, -0.27697354555130005, -1.6978510618209839, -1.0793651342391968, -0.25173622369766235, 0.60561603307724, -0.8832810521125793, -1.6547759771347046, -0.36192384362220764, 1.6661521196365356, 0.2407219558954239, -0.05646461620926857, -0.09659203141927719, 1.1327102184295654, -2.617093801498413, 0.29120731353759766, -0.29892945289611816, 0.3103696405887604, 0.9909723997116089, 0.7672681212425232, 0.5734397172927856, -0.3872620761394501, 1.200237512588501, 1.4923334121704102, -0.09023170173168182, 0.8329411149024963, -0.7791665196418762, 0.3491569757461548, 0.055703260004520416, 0.1297331303358078, -0.25501492619514465, -0.22527208924293518, -1.117289423942566, -0.6010312438011169, -0.839391827583313, 0.5896420478820801, -0.7963338494300842, 0.7225012183189392, -0.5554589033126831, 1.9155359268188477, -0.9749179482460022, 0.06808359920978546, 1.7694109678268433, 2.1174709796905518, -0.09053772687911987, 2.367234230041504, -0.6157265305519104, 0.21983742713928223, 0.5627257823944092, -0.8096222877502441, -0.15554383397102356, -0.09996643662452698, 0.041628915816545486, -0.987727165222168, 1.9237110614776611, -0.14478877186775208, -0.17919929325580597, 0.5397284030914307, 0.3155750632286072, -1.1324323415756226, -0.5246254801750183, 1.422612190246582, -1.4680205583572388, 1.812256097793579, -0.5534912943840027, -0.7672568559646606, 0.24611368775367737, 0.22875189781188965, -0.6064105033874512, -0.2134593278169632, 1.338483452796936, -0.19877947866916656, 0.7364718317985535, -0.23564112186431885, -0.7700332999229431, 0.5296033620834351, 0.8999781012535095, -0.9500601887702942, 1.5657511949539185, 0.14404402673244476, 1.11406672000885, 0.5173853635787964, -1.1345477104187012, -0.9544046521186829, 1.2966171503067017, 0.5290640592575073, 0.4549389183521271, -0.2197151780128479, -1.9009265899658203, 2.5622432231903076, 0.9992752075195312, 0.38632291555404663, -0.558921217918396, 1.8210358619689941, 0.698654294013977, 0.2439536601305008, -0.784221351146698, -1.078324794769287, 0.798338770866394, 1.0455002784729004, -0.4286162555217743, -0.516526460647583, 1.7781733274459839, -1.3254737854003906, -0.18870912492275238, 0.38566553592681885, 1.9975855350494385, 1.2033354043960571, 0.7621229290962219, 0.30625176429748535, -1.5307977199554443, -0.5811863541603088, -0.023332519456744194, 0.13821251690387726, -0.44888728857040405, -0.7544696927070618, 1.4240742921829224, 0.4578869640827179, -3.0118765830993652, 1.5295323133468628, 0.40021654963493347, -1.6249088048934937, -1.1130084991455078, -0.9637660980224609, 0.07220057398080826, 0.001944878720678389, 0.9514051675796509, -0.42420127987861633, 0.7481724619865417, 0.12361469864845276, -0.9864717125892639, 0.9396184682846069, 0.8471148014068604, -0.00017280250904150307, 2.017758846282959, 0.9877700209617615, -1.1577781438827515, 1.4636443853378296, -1.324791669845581, 0.06131414696574211, 0.35647398233413696, 0.7506841421127319, 0.4622915983200073, 0.7698400616645813, -0.5427625179290771, -0.6955282688140869, -0.3314322233200073, -1.1296709775924683, 0.10463691502809525, 1.1622977256774902, 0.9419413208961487, 0.32900920510292053, 1.179523229598999, -0.37437954545021057, 1.2217968702316284, 0.46268850564956665, -0.18456831574440002, 0.6195365190505981, -0.41839417815208435, -2.5806379318237305, 1.548103928565979, 0.17691054940223694, 0.36569106578826904, 1.0321071147918701, 0.9726549386978149, -0.025701632723212242, 1.821940302848816, 0.6894099712371826, -0.614971399307251, -1.5949170589447021, 1.3649928569793701, 0.3468441367149353, -1.156535029411316, 1.3591253757476807, 1.5354365110397339, 0.17408588528633118, 1.9672654867172241, -0.856102705001831, -0.4715117812156677, -0.04084683209657669, 0.37159767746925354, 0.4733986556529999, -0.44910815358161926, 0.38595423102378845, -1.1769797801971436, 0.7101022005081177, 0.6486603617668152, 0.09856757521629333, 0.20195181667804718, 0.9280079007148743, -1.1451972723007202, -0.30328527092933655, 0.09936225414276123, 1.183180809020996, 0.008894347585737705, -0.692376971244812, -0.4994259774684906, -0.4231047034263611, -0.04230737313628197, -0.4590129852294922, -0.8050873875617981, -0.06820644438266754, 0.6842803955078125, 0.474614679813385, 0.1323900669813156, 0.5470903515815735, 1.3825769424438477, -0.9080740213394165, -0.7150264978408813, 1.0454621315002441, 0.8337656855583191, 0.016147606074810028, 0.9976200461387634, 0.7920204997062683, 1.5207860469818115, -0.1741061508655548, 1.2795283794403076, 0.08781485259532928, -1.7791794538497925, -0.19701527059078217, 0.6878353953361511, 0.3727189004421234, -0.3654036223888397, -0.39731019735336304, -0.12086910009384155, -0.7030537128448486, -0.4912247955799103, 2.104295253753662, 0.9149892926216125, 0.26133987307548523, -0.3693021237850189, 1.4314262866973877, -0.6288065314292908, -1.4704012870788574, -1.001129150390625, -1.739327073097229, 0.8532500267028809, 0.8886168003082275, 0.978096604347229, 0.6383482217788696, -1.345534086227417, 1.2631504535675049, -0.2668963074684143, -1.9222701787948608, 0.5645552277565002, 0.3603779375553131, 0.5084565281867981, 0.010348279029130936, -0.5446310043334961, 1.1333853006362915, -1.4801605939865112, 0.7152016758918762, 1.4096271991729736, 0.5236228108406067, 0.9453620910644531, 0.015262517146766186, -0.48557335138320923, 0.04992007836699486, 0.21648074686527252, 0.06640840321779251, -1.3376861810684204, 0.5036645531654358, -0.7530437707901001, 1.7867400646209717, 0.13993355631828308, -0.3729262948036194, 0.623073935508728, 1.4505103826522827, 0.5808040499687195, -0.341052383184433, 1.7416166067123413, -1.2884716987609863, 0.3051370680332184, -1.0314868688583374, -0.17374329268932343, 0.5693657398223877, 0.7767308950424194, -2.112903118133545, -1.3693804740905762, 0.14643514156341553, 0.7222772240638733, 1.5281504392623901, 0.11678852885961533, 1.141510009765625, -0.2549701929092407, 0.3574441373348236, 0.08637981116771698, -1.2214624881744385, -0.14466732740402222, 0.9342412352561951, 0.08957517147064209, 0.2099749743938446, -0.8748605847358704, 1.748484492301941, -1.4397038221359253, 1.0145870447158813, -0.8627603650093079, 1.0517325401306152, 0.139661967754364, 1.2880784273147583, 1.6156362295150757, 0.31716349720954895, 0.4844503104686737, -0.2933485507965088, 0.942440927028656, 1.6433621644973755, -1.629422664642334, -0.6513826251029968, -1.074816346168518, -1.5718507766723633, 0.2688726782798767, -1.236841082572937, -0.06037820875644684, -0.42840635776519775, 1.188202142715454, -1.1249514818191528, -2.620835542678833, -0.41765299439430237, 0.6008936762809753, -0.6819766759872437, 1.8811596632003784, -0.938782274723053, -0.4885336756706238, 0.20135366916656494, 0.5643932819366455, -0.14777901768684387, 0.853796660900116, 1.0796747207641602, 0.46688133478164673, 1.0330393314361572, 0.3855304718017578, 0.25271010398864746, -1.2533245086669922, 0.5503380298614502, 0.7272163033485413, 0.2878356873989105, -0.08775049448013306, 0.6580901741981506, -0.17635396122932434, -0.9292153716087341, -0.6050847172737122, -0.5025877952575684, -1.2343039512634277, 0.8753949403762817, -0.7602372765541077, 1.3018732070922852, -0.5862370729446411, 1.0889670848846436, 0.9966816902160645, -0.43518364429473877, -0.8220656514167786, -0.3035435676574707, 0.38310039043426514, 0.7203548550605774, -0.04050958901643753, 1.3591073751449585, -1.0782840251922607, 0.5827373266220093, -0.7902278900146484, -0.30236586928367615, 0.8703054785728455, -0.6569722294807434, 1.3489207029342651, -0.8907406330108643, -0.8611248135566711, -2.0531020164489746, 0.8631144762039185, -0.5685080289840698, -0.05766003206372261, 1.5166056156158447, 0.3095124065876007, -1.532046914100647, 0.6783813238143921, 0.11715948581695557, -0.6138188242912292, -1.6953325271606445, 0.6688541173934937, -0.8297943472862244, -1.2495415210723877, -1.3368470668792725, 0.7329172492027283, 0.4937458336353302, -0.29343417286872864, 0.21518774330615997, -0.6772072911262512, 0.5371533632278442, 0.8604109287261963, -0.39598169922828674, 0.8316941261291504, -0.03506974130868912, 0.14966870844364166, 0.11705953627824783, -0.1791810840368271, -1.2528101205825806, -0.8865372538566589, -1.4815192222595215, 0.7453941106796265, -0.4892510175704956, -0.9865103363990784, 1.6255238056182861, -0.8197129964828491, -0.1620860993862152, -0.6529216766357422, -0.3426429033279419, 0.7420026063919067, -0.9142001867294312, -2.5368642807006836, -1.766720175743103, -1.3019697666168213, 2.9810776710510254, -0.05693480744957924, -0.033623263239860535, -0.5387619137763977, 0.36210566759109497, -1.4215545654296875, 0.959055483341217, 0.9998051524162292, -0.523608922958374, 0.9715539813041687, 0.732005774974823, -0.5361343622207642, -1.3757959604263306, 0.8174352049827576, 0.040852487087249756, -0.5912910103797913, 0.37682297825813293, -0.1456681191921234, 0.3988726735115051, 1.3488810062408447, -0.07368248701095581, 0.5453134179115295, 1.099496603012085, -0.01592729240655899, 0.6881377100944519, 0.21313907206058502, -0.45740020275115967, -0.5322403311729431, -0.9098258018493652, 0.5416622757911682, -0.1959460824728012, -0.5991101861000061, 0.36181971430778503, -0.4488225281238556, 0.1948205977678299, 0.3832000195980072, -1.0277714729309082, -0.437120646238327, -1.3898415565490723, -0.17386476695537567, -0.8747072219848633, 1.018419623374939, -0.21221019327640533, -0.7445228695869446, 0.5647539496421814, -0.37855127453804016, 1.7747324705123901, 1.2103732824325562, -0.45156288146972656, 1.6175960302352905, -1.3707269430160522, 0.6320497989654541, -0.12339652329683304, -0.8605740666389465, -0.30048587918281555, -1.0647002458572388, -0.020779069513082504, 0.8804934620857239, -0.017345726490020752, 0.6471514105796814, -0.2798091173171997, 0.90418940782547, -0.3438398241996765, -2.240248680114746, 1.0579458475112915, -0.09377509355545044, 0.08046964555978775, 0.14901043474674225, -1.0493779182434082, 0.7412483096122742, 0.8308190107345581, -0.4244251251220703, -1.2096515893936157, 2.1882524490356445, -0.9589714407920837, -1.0776985883712769, -1.4531266689300537, -2.794938087463379, -0.03241986036300659, -0.20105282962322235, -1.1330797672271729, -0.9181089401245117, 0.3575810194015503, 1.8224188089370728, -0.6544150710105896, 0.5791228413581848, -0.7488601803779602, -1.6680241823196411, -0.007533828262239695, -1.20229172706604, -0.9645294547080994, -0.488627552986145, -1.6649022102355957, 2.3386106491088867, -1.732269525527954, -2.3184103965759277, -0.2768023610115051, 1.8224292993545532, -1.4581151008605957, 0.10636541992425919, -0.028505170717835426, 1.731149673461914, 0.632470965385437, -0.26301005482673645, -0.7741739749908447, 2.0825159549713135, -0.6485567092895508, -1.2684345245361328, 1.4912748336791992, -1.0814132690429688, -1.8148834705352783, 0.454987496137619, 0.4919317364692688, -0.2704556882381439, 0.07111022621393204, 0.9832704663276672, -0.325893372297287, 0.5981490612030029, -1.336234450340271, -1.7614997625350952, -0.7367975115776062, -0.5164586305618286, -0.1214325800538063, 0.2733492851257324, 1.219767689704895, 0.24828538298606873, -0.4102267920970917, -0.5100847482681274, -0.4588722586631775, -1.1193668842315674, 0.9696239829063416, 0.4154045283794403, -1.1491304636001587, -0.5054792165756226, -1.0872594118118286, 1.4058842658996582, 1.1188719272613525, -0.36593908071517944, -0.6528586745262146, -0.5076179504394531, -1.040598750114441, 1.0248512029647827, -0.21169261634349823, 0.7486112713813782, 0.3105832040309906, 0.36016371846199036, 0.23703281581401825, -0.5498074293136597, -0.09956490248441696, -2.345303535461426, -0.11340164393186569, 0.7860462069511414, 0.11408788710832596, 0.04373885318636894, 1.5045173168182373, -0.23692388832569122, -0.8387901186943054, 1.5060628652572632, 1.548410177230835, 0.03659424930810928, 0.2984393239021301, 0.2662474513053894, 1.0596896409988403, -0.5786288976669312, 0.09646867960691452, 1.6678674221038818, 0.8264842629432678, 0.9557877779006958, 0.5027998685836792, -0.7279250025749207, -1.1294952630996704, 0.5547451972961426, -0.32344064116477966, -1.6946052312850952, 1.1662591695785522, -0.9591593146324158, 0.26179003715515137, -1.1214419603347778, -0.6540439128875732, 0.9177324175834656, -0.7352147102355957, 1.4289809465408325, -0.11261846870183945, 0.6341462135314941, -0.2583448588848114, 1.3166842460632324, -1.3501662015914917, -2.2679667472839355, -1.541693925857544, 0.25261786580085754, 0.5727915167808533, -1.6453337669372559, -0.49529364705085754, 0.9631381630897522, -0.577589750289917, -0.001880671246908605, -0.28378087282180786, -1.1370271444320679, 0.865591287612915, 0.0657181441783905, -0.8383736610412598, -1.572248101234436, -1.2478365898132324, -0.0655226930975914, 0.11733030527830124, 0.4875803589820862, 2.0353198051452637, 0.554092526435852, -1.3151615858078003, -0.6597627401351929, 1.3414008617401123, -1.1530343294143677, 0.7899559140205383, -2.999048948287964, -0.29743868112564087, -0.5586562156677246, -0.43521350622177124, -0.17927230894565582, -0.5033760666847229, 0.3473668098449707, 0.9056574106216431, 0.834801197052002, 1.065910816192627, -2.051717519760132, 0.94879150390625, -0.5420331358909607, 0.4844502806663513, -0.34159693121910095, -0.5243111848831177, 0.054710231721401215, 0.8983127474784851, 1.0695488452911377, 2.4924888610839844, 1.098825216293335, -1.0717345476150513, 0.7700060606002808, -1.9818141460418701, -1.451445460319519, -0.35027778148651123, 0.2714901566505432, 0.3300662636756897, -0.20250168442726135, -0.5316126942634583, -0.5606153011322021, 0.4821752905845642, 2.2336690425872803, 1.2269463539123535, 1.6073299646377563, -2.0251362323760986, 2.0927300453186035, -1.2331264019012451, -2.34082293510437, -0.882117509841919, 0.49585163593292236, -0.1525113880634308, 0.33062243461608887, -0.0669335126876831, -0.7423924803733826, -0.3326307237148285, -1.8414145708084106, 0.4525259733200073, 0.162692129611969, -1.8251280784606934, 1.767850637435913, -2.4887518882751465, 0.9456866979598999, -0.15621809661388397, 0.04070242494344711, -0.24282483756542206, 2.7982513904571533, -0.45491352677345276, 0.4225630462169647, 0.5444594025611877, -0.39589324593544006, 0.9642841815948486, 1.0929113626480103, 0.8893164992332458, -0.5173020958900452, -0.09509682655334473, -0.7159184217453003, 0.4433456361293793, 0.09987955540418625, -0.3282198905944824, 0.2224040925502777, 0.6980117559432983, -0.6758346557617188, 0.1456204503774643, -0.6723091006278992, -1.284735083580017, -0.9312187433242798, -0.2319173514842987, 1.068619728088379, -1.5829733610153198, -0.00562896765768528, -1.5291578769683838, 0.07581725716590881, -1.757549524307251, 1.874890923500061, 0.5367132425308228, 0.5972192883491516, -0.3306456208229065, 0.25058019161224365, -0.4070563018321991, -0.1854899674654007, 2.101278305053711, 0.21935521066188812, -1.886864423751831, -0.6133335828781128, 0.9665523171424866, 0.21586725115776062, -0.47953155636787415, -0.11843077093362808, 1.1243948936462402, 1.5413730144500732, -0.06444068998098373, -0.5537279844284058, -0.390379935503006, 2.092195749282837, 0.006582585163414478, -0.8976398706436157, 0.22242441773414612, -0.09035760909318924, -2.298408269882202, -0.02879161946475506, 1.1546412706375122, -0.14174264669418335, 0.6144565343856812, -0.4282641112804413, -0.9712727069854736, 1.2431010007858276, -0.6628921627998352, 0.25517114996910095, -1.3702698945999146, 0.1605670005083084, 0.1069294661283493, -0.9423513412475586, 0.9723874926567078, -1.2822043895721436, -0.4500068426132202, 0.6525635719299316, -0.2967623770236969, 0.29762786626815796, -1.3864691257476807, -0.3771161139011383, 0.24108490347862244, -0.7091830372810364, 1.3744336366653442, 0.6412333250045776, -0.8304675221443176, -0.8553087711334229, -1.013100504875183, 0.35535529255867004, -0.1371152400970459, 0.06278011202812195, 0.018961716443300247, 0.03874348849058151, 0.24962197244167328, 0.35907337069511414, -0.8904857039451599, -0.7101480960845947, -0.5413380861282349, 0.16910740733146667, -0.6928567886352539, -0.18686020374298096, -0.5886000394821167, 0.3222713768482208, 0.6705791354179382, 0.7956743240356445, -1.1558865308761597, 1.0145646333694458, -0.9721189141273499, -1.2486748695373535, 0.23548635840415955, -0.6891627311706543, 0.7403637170791626, -0.8636128902435303, -0.2915220260620117, 0.18942323327064514, 1.4401798248291016, 0.24621468782424927, 1.515451431274414, -0.4687749147415161, 0.5750887393951416, 0.9917394518852234, -1.4671480655670166, -0.25775769352912903, -0.5668020844459534, 0.001119022723287344, 0.28029218316078186, -0.24971246719360352, 1.1434131860733032, 0.1023499146103859, 0.3243010938167572, -1.5668842792510986, 0.31151995062828064, 0.8931189775466919, 1.7984724044799805, 1.1848106384277344, -0.6572505235671997, -2.4276835918426514, 1.4166276454925537, -0.6643281579017639, 0.27385810017585754, -1.2799433469772339, 2.3439435958862305, 0.6861213445663452, -1.5979770421981812, -1.4054087400436401, 2.107271671295166, 0.5652371644973755, -0.0200639795511961, 0.011640280485153198, 0.9665629863739014, -0.24055415391921997, 0.9581918716430664, -0.46519577503204346, 0.8239394426345825, -1.570083498954773, 0.5861823558807373, -0.5989580750465393, 0.2521953880786896, -0.2836942672729492, 1.1151987314224243, -0.1922784149646759, 0.16219446063041687, -1.321660041809082, 1.138641119003296, 0.4187581539154053, -0.8338271379470825, 0.17345765233039856, 0.19665835797786713, -1.4008610248565674, 0.7089039087295532, 1.2493555545806885, 2.137465715408325, 0.8944408893585205, 0.3752498924732208, 1.1949167251586914, 0.31186532974243164, 1.06577730178833, 0.6835584044456482, -0.5797580480575562, 0.4658339023590088, -0.6206808090209961, -0.1396433264017105, -0.6566879153251648], [1.059212565422058, 1.3389376401901245, 0.08253529667854309, 0.25562477111816406, -1.191909909248352, 0.6106672286987305, -0.08294800668954849, 0.44857680797576904, 0.47566285729408264, -0.7271450757980347, -0.39578506350517273, -0.40347757935523987, 1.5528194904327393, 1.9454900026321411, 2.030329942703247, 0.6888837814331055, -1.6973663568496704, 1.2245759963989258, -0.32386621832847595, -0.6663581728935242, 0.6563960313796997, -1.5111397504806519, -0.438228577375412, -2.577345848083496, -1.5147697925567627, -0.3226606845855713, 0.3888701796531677, -1.3752208948135376, -0.19286464154720306, 0.4643232822418213, 0.7723897099494934, 1.073923945426941, -0.7630752325057983, 0.7180094718933105, 0.9057087302207947, -1.1319226026535034, 0.4662284851074219, 0.08202219754457474, 0.18930698931217194, -0.3575515151023865, 1.2817223072052002, -1.7338368892669678, 0.2552184760570526, -0.16323834657669067, -1.399072527885437, 1.732037901878357, 0.9771982431411743, 0.22375623881816864, 0.23591972887516022, -1.794160008430481, 0.15194089710712433, 0.8639547824859619, 1.1079622507095337, 1.0092980861663818, 0.9759886264801025, -0.4427262544631958, 1.9264675378799438, 0.2840156555175781, 1.4062788486480713, -0.829595148563385, 0.42539146542549133, 0.3334885239601135, 0.3958222568035126, 1.32745361328125, -1.1998419761657715, 0.786333441734314, 0.4659118056297302, 0.9498106837272644, -0.19150987267494202, 0.8598114252090454, -0.11071688681840897, 0.15116232633590698, 0.4350481331348419, -1.5061851739883423, 1.8036327362060547, 1.9909590482711792, 0.8516986966133118, 0.42883577942848206, 1.1536225080490112, 0.8819730281829834, -0.4560006260871887, -0.715194821357727, 1.1221847534179688, -0.4464016556739807, 0.8611075282096863, 1.1201610565185547, 1.2693345546722412, 1.3989899158477783, -0.07242186367511749, 0.9377720952033997, 1.1566828489303589, -0.24350422620773315, -0.423323392868042, 0.6021215319633484, -1.5687228441238403, 0.140711709856987, -1.044104814529419, 1.0350337028503418, -1.2075608968734741, 0.8826953172683716, 1.592253565788269, 0.8143149018287659, 0.5504626631736755, -1.0755163431167603, 0.07120867073535919, 0.8602908253669739, -0.19465577602386475, 0.11376593261957169, 0.3308778405189514, 0.4152497351169586, -1.4420901536941528, -0.896124541759491, -0.3234213888645172, -1.3985146284103394, -0.2579389214515686, -1.1289844512939453, -1.0789355039596558, 0.7244104146957397, -0.5722216367721558, 0.23003467917442322, -1.1745914220809937, -1.2187126874923706, -2.0277793407440186, -1.636325716972351, -0.7491443157196045, 0.42820826172828674, -0.29868578910827637, -0.01812312938272953, 0.6051491498947144, -0.9782002568244934, 1.2440719604492188, -0.23688410222530365, 2.00456166267395, -1.137776494026184, -1.4979832172393799, 1.535670518875122, -0.08188852667808533, 0.24771913886070251, 1.367098093032837, -0.47942396998405457, 0.9922962784767151, -0.7026064395904541, -0.40500712394714355, 0.16217538714408875, 1.9997614622116089, -2.2147328853607178, -1.2070201635360718, 0.2805119752883911, -0.38951581716537476, 0.897924542427063, -1.151105523109436, 2.598283529281616, 1.491777777671814, 0.9556702375411987, 0.833768904209137, -0.785922110080719, 0.6290461421012878, -1.899979591369629, 0.8970984816551208, -0.44139060378074646, -0.43624958395957947, 1.985216498374939, -2.4025585651397705, -0.8680285811424255, 0.2584243416786194, 0.16265743970870972, -0.43287065625190735, 1.1997342109680176, -0.8422879576683044, -0.24173487722873688, -1.6149497032165527, 1.8908928632736206, -2.0446646213531494, -0.08021356165409088, 0.08399725705385208, -0.9283340573310852, -0.9463295340538025, -0.38177749514579773, 0.3060855269432068, -1.0799164772033691, 0.08676522225141525, -0.2794643044471741, 2.4920036792755127, 1.067292332649231, 0.5389144420623779, 0.22888964414596558, -0.14475668966770172, -0.44779646396636963, 0.1150127723813057, -0.6633053421974182, -0.8679249286651611, -0.8938847780227661, -1.935911774635315, 0.791266143321991, -0.9063370823860168, -0.22793622314929962, 0.34000810980796814, 0.671413779258728, 2.3720831871032715, 0.25110870599746704, -0.4532492160797119, 0.2572128176689148, -1.3612090349197388, -0.39964407682418823, -0.09948766231536865, -1.1593639850616455, 0.20725736021995544, -1.0287030935287476, -0.9540285468101501, 1.827188491821289, 1.7592779397964478, 0.8105986714363098, 0.34523075819015503, -0.4162265360355377, 1.1830589771270752, -0.5094231963157654, 1.0549870729446411, 0.48790910840034485, -1.3248693943023682, 0.3448847234249115, -0.9090186953544617, -1.118430495262146, 2.4865362644195557, -0.6928601264953613, 0.14055004715919495, -0.3167911171913147, 0.8703120350837708, 0.878216564655304, 0.2962127923965454, -0.6808634400367737, 1.107170820236206, 0.6438888311386108, 0.37484535574913025, 0.3312833607196808, 0.14102192223072052, 0.21206113696098328, 1.8747036457061768, 0.14621701836585999, 0.4390515387058258, -0.13347718119621277, 0.4535995423793793, 0.09662000834941864, 0.4140636920928955, -0.7401403188705444, -1.3032450675964355, 1.4856338500976562, -1.7237550020217896, 0.0861009880900383, 1.0621486902236938, -0.00046261565876193345, 0.36949047446250916, 1.2813379764556885, 0.5971274971961975, 0.595590353012085, -1.24652099609375, -0.6082573533058167, 0.09625004231929779, 0.46832966804504395, -0.7798737287521362, -0.9090110063552856, 2.2627365589141846, 0.12788335978984833, 0.6750051975250244, 0.24672362208366394, 0.11896400153636932, -1.8127745389938354, 0.4648490846157074, 0.815276563167572, 0.8820705413818359, 0.25429001450538635, 0.17308689653873444, -0.37219125032424927, -0.16826142370700836, 0.17970699071884155, -0.2814544439315796, -1.6131004095077515, 0.5397041440010071, 0.9004060626029968, 0.06449297815561295, 1.7399615049362183, -1.4633371829986572, -1.1388533115386963, 0.05417101830244064, 0.7969108819961548, 0.2742388844490051, 0.1625719666481018, 0.7066174745559692, -0.5148458480834961, 0.7577124238014221, -0.7353946566581726, -0.5156369805335999, -0.2034885436296463, 0.181769460439682, 1.9408880472183228, -2.119960308074951, -0.11257662624120712, 1.1691603660583496, -0.49865278601646423, 0.7933559417724609, 0.16829591989517212, 1.2642496824264526, 0.6054770350456238, 1.2846559286117554, 1.3826414346694946, -0.7220174074172974, 0.5431389212608337, -0.037082452327013016, -1.242580533027649, 0.40222570300102234, -0.2483615279197693, -0.5250667929649353, -0.3766983151435852, 1.0394748449325562, -0.7872346043586731, 0.0852014347910881, 0.5254368782043457, -1.1790982484817505, -0.3018980920314789, 2.7894999980926514, 0.0713941901922226, -0.4075075387954712, -1.0947622060775757, -0.03235038369894028, 1.0471560955047607, -2.3126258850097656, -0.9382855296134949, -1.1251658201217651, 0.290637731552124, 1.6838772296905518, -0.2186729460954666, -1.3024535179138184, 0.2333076447248459, -0.13019803166389465, 0.1357385665178299, 0.6595969796180725, 0.2872070372104645, -0.16096444427967072, -0.6231917142868042, -0.6617486476898193, 1.6832016706466675, -0.28125834465026855, -0.8136761784553528, -0.9211754202842712, -0.3408196270465851, -0.5129771828651428, 1.3297899961471558, 0.1858990639448166, -0.2969207167625427, 1.6109287738800049, -0.2651609480381012, -0.40899601578712463, -0.17947795987129211, -1.1013997793197632, 0.9739558100700378, 0.14630651473999023, 2.364504814147949, -0.26704683899879456, 0.16342420876026154, 0.8266090750694275, 2.164067268371582, 0.5132476687431335, -0.5236198306083679, 0.10881439596414566, -0.8166772127151489, -0.018455147743225098, 0.4099402129650116, 0.5240503549575806, 0.9871604442596436, 0.24413955211639404, -0.29792892932891846, -0.08398599177598953, -0.38714316487312317, -1.3290669918060303, -1.1939589977264404, 0.14409048855304718, -0.43483462929725647, 0.7382732629776001, -0.14255087077617645, 0.08826149255037308, 0.7594922184944153, -0.12602487206459045, 0.4652203917503357, 0.41478264331817627, 0.8531142473220825, 2.0461039543151855, 0.12774255871772766, -0.4498438239097595, 0.941868007183075, -0.4130544066429138, -1.04990553855896, -0.2550131380558014, 1.458429217338562, 0.2882983684539795, -0.5737755298614502, 0.34624966979026794, 0.572318971157074, 0.46699100732803345, 0.26509684324264526, 0.3575589060783386, -0.5422533750534058, 0.32776781916618347, 0.01825084164738655, -1.2967408895492554, 0.7181040644645691, -0.11790604889392853, -0.40333104133605957, -0.015712561085820198, 0.7614638209342957, -0.5381699800491333, 2.4207913875579834, -0.5969537496566772, -1.9939090013504028, 0.013388002291321754, -0.02450275793671608, 0.42862558364868164, 0.8631728291511536, -0.039293322712183, 1.0043349266052246, -0.16929005086421967, 0.6109304428100586, -0.23115570843219757, 1.3355520963668823, -0.3221322000026703, 0.7462409734725952, 1.3127812147140503, -1.8738833665847778, 0.26002854108810425, 0.0036292364820837975, 1.678290605545044, 0.10973572731018066, -0.32402509450912476, -1.1892495155334473, 1.5702476501464844, -0.8085526823997498, 1.1397373676300049, 0.39778316020965576, 0.3072260022163391, -1.1883608102798462, 0.8348992466926575, 1.036087989807129, 0.4325212836265564, -0.832542359828949, -0.08237594366073608, -0.1905192732810974, -0.025652915239334106, -0.3162347674369812, 0.582351803779602, -0.409123957157135, 1.0999361276626587, 0.2753726541996002, 0.05487634986639023, -0.8807302117347717, 0.31191524863243103, 0.14773781597614288, -0.2346230000257492, -1.5971022844314575, 1.0835936069488525, 0.45119044184684753, -0.798689603805542, 0.7724622488021851, -0.5279107093811035, -0.665534496307373, -0.9409092664718628, 0.624966025352478, 1.4588090181350708, -0.938643753528595, 1.4083945751190186, -0.16166779398918152, -0.21411047875881195, -0.43486136198043823, 0.5965160727500916, -0.6526496410369873, 0.31560423970222473, 0.1582769900560379, 0.12099349498748779, -0.23695850372314453, -0.7300111055374146, -1.0568439960479736, -0.22840581834316254, 0.0413832925260067, -0.044521868228912354, -1.3407648801803589, 0.6518452763557434, 0.6366692781448364, -1.472214698791504, 0.6435574889183044, -0.6162648797035217, -0.621140718460083, 0.3768623471260071, -0.35721248388290405, -1.6432644128799438, -0.8301151394844055, 0.12092648446559906, 0.12418780475854874, -0.5925859212875366, -0.6427463293075562, 0.10172642022371292, -1.447368860244751, 0.34302404522895813, 0.790666937828064, 0.37931305170059204, 2.568082809448242, -0.0139312744140625, -0.3141520321369171, -0.4504520297050476, -0.8334058523178101, -0.3184109628200531, 1.1914801597595215, -0.22814308106899261, 0.41565442085266113, 1.720276117324829, -0.21273170411586761, 1.4962190389633179, 1.0944548845291138, 0.2266267091035843, 0.24979260563850403, 0.844782292842865, -0.4856737554073334, 0.4277288019657135, 0.6526744961738586, -0.7862980961799622, -1.238268256187439, 0.11644255369901657, -0.35089701414108276, 0.6705647110939026, 0.5020736455917358, -1.0694891214370728, 0.06361964344978333, -0.5886600613594055, 1.2545095682144165, 1.4549654722213745, 0.7380911707878113, -1.2809768915176392, 1.6220353841781616, -0.632450520992279, -0.7187554240226746, 0.1928907185792923, -1.0166717767715454, -0.5178218483924866, 0.04258520156145096, -0.4939122200012207, -0.08549392968416214, 0.08432445675134659, 0.556664228439331, 0.542445719242096, -0.7403191328048706, -1.2240700721740723, -0.680647075176239, -0.3850077688694, -1.6419013738632202, -2.4285919666290283, -2.315359115600586, -0.1902705430984497, 1.62594473361969, -0.426082044839859, -0.7886940240859985, -1.561127781867981, 0.5050137042999268, -0.7720105648040771, -0.8224824666976929, 0.7361191511154175, -2.3118371963500977, -1.972008466720581, 1.181344985961914, -1.375485897064209, 1.3765082359313965, 0.3367175757884979, -1.070083737373352, 1.0083404779434204, -0.526727557182312, -0.4806469678878784, 0.8850599527359009, 0.12488457560539246, 1.421594262123108, 1.4832050800323486, 0.3022118806838989, 1.5631043910980225, -0.08561550825834274, -0.44387587904930115, -0.009998179040849209, 0.5082250833511353, 1.6776236295700073, -0.7654439806938171, 0.29640746116638184, 0.6254635453224182, -0.9279266595840454, 0.6462669968605042, 0.2817331850528717, -0.5956306457519531, 1.0687175989151, 0.34489983320236206, -0.6568055748939514, 0.8018987774848938, -0.28223881125450134, -1.5791863203048706, 1.0905600786209106, 1.2584896087646484, -0.15807795524597168, 0.20573163032531738, -0.7891442775726318, 0.027240199968218803, -0.7706005573272705, -0.06303676217794418, -0.5796838402748108, -0.554553210735321, -1.381147027015686, -0.2061629593372345, 1.9351387023925781, 1.2832462787628174, 2.9918134212493896, -0.5990224480628967, -0.45230618119239807, 1.5438928604125977, -1.123407006263733, -1.0216842889785767, 1.997214913368225, 1.062090277671814, -2.438112497329712, -0.046192318201065063, -1.650177240371704, -1.0466564893722534, -0.363800585269928, -1.0481462478637695, 1.0700790882110596, -0.005006539635360241, 0.006038929335772991, -0.6407231092453003, -1.5840626955032349, -0.702613890171051, 0.6190275549888611, -0.47953879833221436, 1.52497398853302, 1.4318798780441284, -0.723753035068512, 1.2276992797851562, 0.5776987671852112, 1.2683324813842773, -1.9007620811462402, 0.716437578201294, 0.41120997071266174, 0.21432805061340332, -0.5821054577827454, 0.6354018449783325, -0.7057262659072876, -1.0863515138626099, 1.2294254302978516, 1.5534013509750366, 0.1657463163137436, -0.4288662075996399, -0.1735549420118332, 1.6998769044876099, -0.6902221441268921, -0.4131854772567749, 0.7386597990989685, -1.7396329641342163, 0.07191132009029388, -0.01287139393389225, -0.3353091776371002, 0.401358425617218, 2.147489547729492, 1.7445881366729736, -1.6610002517700195, -1.3589950799942017, 0.17032776772975922, -0.1862759292125702, -0.21319283545017242, -0.5152016282081604, 0.5621581077575684, -0.39019539952278137, -0.979062020778656, 0.6210891604423523, 0.11814828217029572, -0.0358789823949337, -0.7117965221405029, -1.6489285230636597, -0.46622446179389954, 0.7896654009819031, -0.13812647759914398, 1.2447056770324707, 0.6351631283760071, -0.9136048555374146, -0.41868072748184204, -0.3887384831905365, -0.1975129395723343, -0.7976618409156799, 1.3934059143066406, -0.6662936806678772, 0.2918875217437744, -0.6732556223869324, -0.7364283800125122, -0.3931426703929901, -1.5437016487121582, 0.7541689276695251, -0.40055105090141296, 0.09235242009162903, -1.394845962524414, -1.1482123136520386, 0.43179965019226074, 0.6641283631324768, -1.5433789491653442, 1.847663164138794, -1.9044177532196045, -0.07000355422496796, 0.1756436973810196, -0.20129141211509705, 0.6776845455169678, -1.049782156944275, 0.43485966324806213, -0.9158062934875488, -0.7605723142623901, -1.1496634483337402, 2.366096258163452, 1.1258316040039062, -0.8113886713981628, -1.2029551267623901, -0.9085767269134521, 1.9496779441833496, -0.015634292736649513, -2.0794403553009033, -1.8080753087997437, 0.8044267892837524, -0.19474512338638306, 1.9390393495559692, 1.3076810836791992, -0.46191462874412537, 0.8044496178627014, 0.2874489426612854, 0.4422696530818939, -0.9834530353546143, 0.7020966410636902, 0.5582608580589294, -1.4456818103790283, 1.069458246231079, 1.3223427534103394, -0.02092941477894783, -0.3292127549648285, 0.19858722388744354, 1.0572173595428467, 0.3836916387081146, -0.6384584903717041, 0.7224353551864624, -1.6966713666915894, -1.5840122699737549, 0.4021015167236328, 0.4456266760826111, 0.9614519476890564, 1.1773841381072998, 0.981812596321106, -1.3634551763534546, 0.6340633630752563, 1.6639870405197144, -1.66722571849823, 0.36458656191825867, 0.7636050581932068, 0.33904629945755005, 0.8286845684051514, 0.04724077135324478, 1.846416711807251, 0.31116360425949097, -0.28309863805770874, 0.415375679731369, -0.8457618355751038, -0.14707309007644653, 0.8763212561607361, -0.2826605439186096, 1.3528931140899658, -1.3241223096847534, -2.366387367248535, 1.4006868600845337, 0.71913743019104, -0.6276113986968994, 0.5545619130134583, -0.6079356670379639, 1.0339149236679077, 1.1839808225631714, -0.232164204120636, 0.8450995683670044, -0.9290523529052734, -1.202474594116211, 0.7562766671180725, 0.16821645200252533, -1.2575154304504395, -0.16451796889305115, -0.2568276822566986, -0.4686277210712433, 0.08118602633476257, 0.13193508982658386, 1.6221888065338135, -0.22812677919864655, -0.014787810854613781, -0.8527926802635193, -0.9327584505081177, 0.037349894642829895, 1.6956831216812134, -0.35658207535743713, 0.34868040680885315, -1.4976271390914917, 0.6815726161003113, 0.04906008392572403, 0.5249184370040894, -1.095574975013733, 0.6186783313751221, -1.0215009450912476, -2.2741878032684326, 2.211723804473877, 1.683253288269043, 1.116795301437378, 0.40064477920532227, -0.1208718940615654, -0.41678592562675476, -1.3816187381744385, 0.8810774683952332, -2.601489782333374, 0.7395788431167603, 0.8238602876663208, -0.3141550123691559, 0.036787841469049454, 0.37981387972831726, -0.2350352555513382, -0.2587403953075409, 0.1640772968530655, -1.0657711029052734, -0.47913381457328796, 1.8635298013687134, -1.0932904481887817, -0.2723340094089508, -0.746730625629425, -0.9656571745872498, 0.7573045492172241, -0.9548355340957642, -0.21027638018131256, -0.22387422621250153, -0.021325696259737015, -0.5855981111526489, -0.8753075003623962, 0.40169769525527954, 2.3390204906463623, 0.2670205533504486, -0.28888481855392456, 0.2859136462211609, 0.6047398447990417, -0.255727082490921, 1.0023412704467773, -0.06940913945436478, 0.7184996604919434, -0.5722175240516663, -0.24770388007164001, -1.3933289051055908, 0.4703696072101593, -0.8929013609886169, 0.9199756979942322, 0.7535114884376526, 0.8276296854019165, 0.4828793406486511, -0.0018974802223965526, -0.2057805210351944, 0.08011738210916519, 0.9874117970466614, 0.8827824592590332, 0.6766345500946045, -0.7336093187332153, -0.17951475083827972, 0.13038939237594604, 1.0703448057174683, -1.3091181516647339, 0.07633349299430847, 0.8844320178031921, 1.7538877725601196, -0.12215592712163925, -0.12283109128475189, -0.23179620504379272, -0.24704311788082123, 1.3679317235946655, -0.5702211260795593, 0.8228952288627625, 2.8728301525115967, 1.6117271184921265, 0.3391067087650299, 0.7470259666442871, -2.072558641433716, -0.6406129598617554, -1.6775332689285278, -0.35150590538978577, 1.613752841949463, -0.25936391949653625, 1.3884613513946533, -2.177541971206665, -1.0580610036849976, 0.3823103904724121, 0.9465757012367249, -0.9363223910331726, 1.671126365661621, 0.3550094664096832, 0.10717379301786423, 0.056188005954027176, 0.6989311575889587, 0.7141404151916504, 0.6692646145820618, -1.1109740734100342, 1.3598666191101074, -0.26658105850219727, 0.6427993774414062, -1.1613993644714355, -1.1388646364212036, -0.1788836568593979, -0.567592978477478, 0.3671240508556366, -0.15608999133110046, -0.4603646397590637, -0.07355594635009766, -1.692453145980835, -0.5870540738105774, 0.5814936757087708, 0.43655723333358765, -2.350701332092285, -0.8619965314865112, 0.9843302965164185, -0.08906968683004379, -0.6553124189376831, 0.07177301496267319, 1.156649112701416, -1.0146747827529907, 0.9891864061355591, -1.666807770729065, 0.06377605348825455, -2.0461156368255615, 0.6674491167068481, -0.7310760617256165, -0.42291194200515747, 1.2107048034667969, 0.15884079039096832, -1.7028424739837646, -1.077805995941162, 0.9075164198875427, 1.635748267173767, -0.3350047171115875, -0.8174912333488464, -1.895473837852478, 0.23102471232414246, 0.5618413686752319, -0.22749914228916168, 1.6869945526123047, -2.373854160308838, -0.8457372188568115, 0.44966551661491394, 0.512769341468811, -1.1626017093658447, 0.7827203869819641, 1.893710970878601, -0.33548131585121155, -0.043254345655441284, -0.5648126602172852, 1.22908353805542, -1.4447057247161865, 0.8000551462173462, -0.5479581356048584, 1.600771188735962, -0.8475792407989502, 0.6598106622695923, 0.48710161447525024, 0.4832218587398529, -1.5442403554916382, -1.4870989322662354, 0.8091948628425598, -0.0612027645111084, 1.3800734281539917, 0.79599928855896, 0.11854447424411774, -1.7240231037139893, -0.9768145680427551, -1.6932560205459595, 0.03194025531411171, -0.34443068504333496, -0.1952800154685974, -0.9622555375099182, 1.678148627281189, -0.18431372940540314, -1.1261917352676392, 0.027933653444051743, -0.7714669108390808, 0.4194177985191345, -0.24319641292095184, 0.8091766238212585, -0.08380362391471863, 0.4234873354434967, 0.5828238725662231, 0.26340821385383606, 1.1690605878829956, -1.760968804359436, 0.10948416590690613, -0.10631158947944641, -2.3328757286071777, -0.837792694568634, 0.6753869652748108, -0.5362834334373474, 0.2089860886335373, -1.0367956161499023, -0.8907995223999023, -0.6653963923454285, 0.42833879590034485, -1.770719289779663, -1.9285004138946533, -2.009892463684082, 0.6066315174102783, 0.6701303124427795, 0.3094346225261688, -0.895051896572113, -1.1561942100524902, -0.9775139093399048, -0.7030975818634033, 0.13154271245002747, 0.31194764375686646, 0.022145120427012444, -0.7447713613510132, 0.06275282055139542, -0.24976840615272522, 0.8472795486450195, -0.3199886381626129, -0.09314141422510147, -0.2712100148200989, -0.9177494049072266, -0.3995814323425293, -0.6732834577560425, 1.5728988647460938], [1.9143236875534058, 0.7479491233825684, -0.19840621948242188, -0.12035068869590759, -0.9943363666534424, -0.8133369088172913, -0.13705609738826752, -0.37230828404426575, -0.05742873623967171, -0.5895076394081116, -1.1744405031204224, -0.5777022838592529, 0.24861203134059906, 0.6938828825950623, -1.9662681818008423, 0.7581473588943481, -0.1289142370223999, -0.28454458713531494, 0.22100742161273956, 0.32003435492515564, -1.3096712827682495, 0.5091708898544312, 0.5499089956283569, 1.2234758138656616, 1.2587661743164062, -0.4261232316493988, -0.9890959858894348, 1.076230525970459, 1.2386443614959717, 1.4409831762313843, -1.468246579170227, -1.2264056205749512, 0.6822635531425476, 0.9823246002197266, -0.6767995953559875, -1.9942958354949951, -0.13630856573581696, 0.47714701294898987, -0.2564094066619873, 1.5908082723617554, -1.8509916067123413, 1.3457653522491455, -1.1926387548446655, 0.5799244046211243, 0.3873157799243927, -2.490841865539551, 0.630705714225769, -0.4671974778175354, -1.1131489276885986, 0.7125232815742493, 0.7759394645690918, -0.3336251378059387, 0.09836813807487488, -0.6502764225006104, 0.13326168060302734, 1.1924515962600708, 0.06250623613595963, 0.48651668429374695, -1.505281686782837, -0.6374874711036682, -0.01433319877833128, 0.789088249206543, 1.0055451393127441, 0.08754389733076096, -0.6884704232215881, 0.7061548233032227, 0.10894611477851868, 0.00816514901816845, 0.4522867798805237, -0.19189924001693726, 0.5809465646743774, 0.008148236200213432, 1.494138240814209, 0.5647149682044983, 1.0509871244430542, 0.8099520802497864, -0.022674597799777985, -0.5547862648963928, 2.157763957977295, -0.6999561190605164, -1.7210334539413452, 0.25669145584106445, -0.18805228173732758, -0.17905062437057495, -0.028506813570857048, -0.26991501450538635, 0.6168480515480042, 3.1636157035827637, -0.3233202397823334, -1.6098017692565918, 0.4241834580898285, 0.9868585467338562, 0.27706825733184814, 1.215491771697998, -0.08806919306516647, -0.8554806113243103, 1.633711576461792, -0.5565075278282166, 1.511373519897461, -1.503414511680603, 0.0815437063574791, -0.7272299528121948, 1.206668734550476, -1.9199823141098022, 2.123079776763916, 0.34645983576774597, -1.794517159461975, 0.5349476337432861, -0.2850421369075775, 0.7576465606689453, -1.3412827253341675, 0.40327581763267517, -0.45638489723205566, -0.2686192989349365, -2.1092724800109863, 0.007548225577920675, -0.6389755010604858, -0.3075779974460602, 0.10326910763978958, -0.13213978707790375, -0.9278815984725952, 1.375835657119751, 0.20496059954166412, 0.07411672174930573, -0.5134694576263428, -1.297584891319275, 1.1323680877685547, -1.1725562810897827, -0.7584813237190247, 0.8289509415626526, -0.36265748739242554, 0.8795139193534851, 1.566940188407898, 1.0340958833694458, -0.10852295905351639, -0.766130805015564, -1.519883155822754, 1.1081929206848145, 0.43432334065437317, 2.413865089416504, 0.09715509414672852, 1.107596755027771, -0.3511578142642975, -0.6280652284622192, 0.23133116960525513, -0.5539915561676025, 1.0187760591506958, 0.16641993820667267, 1.4805420637130737, -0.3465688228607178, 0.3792565166950226, -0.6444024443626404, -0.6943063139915466, -1.4734888076782227, 0.6078344583511353, -0.698976457118988, -0.4153749644756317, -0.43302568793296814, -0.6745871305465698, 0.7681488990783691, 0.2683384120464325, 0.4696042835712433, -1.3183120489120483, 0.8812882900238037, -0.5707665681838989, 1.8180227279663086, 0.8630470037460327, -1.6840085983276367, -0.8763464093208313, 0.6566284894943237, 0.1502952128648758, 0.10483536869287491, -0.07551652938127518, 1.1044294834136963, 0.6869826316833496, 0.7828202843666077, -0.6100203990936279, -1.1188666820526123, -0.14898909628391266, 0.5573429465293884, -1.0935217142105103, 0.03755147382616997, -1.7331501245498657, -1.272658109664917, 0.054607268422842026, -0.3264114558696747, 0.8685488700866699, -1.0445892810821533, -0.14269563555717468, -0.3596407175064087, 0.11454501748085022, -1.0513485670089722, 2.2422430515289307, 0.9831500053405762, -0.21618440747261047, -1.0170633792877197, -0.5334306359291077, 0.5841853618621826, -0.15731464326381683, -0.5141165852546692, 0.012912175618112087, -2.774501323699951, -0.26325392723083496, -0.32632574439048767, -1.5110160112380981, 0.3486970365047455, -1.7619234323501587, 1.6558562517166138, -0.3518473207950592, 0.4868985712528229, 1.01487135887146, -2.071096658706665, 0.7229954600334167, 0.25410592555999756, 1.0258854627609253, 0.3470035791397095, 0.13301457464694977, -0.24510794878005981, 0.13606907427310944, 0.060516055673360825, 1.3938777446746826, -0.3826166093349457, 0.9255977869033813, 0.46494248509407043, 2.784943103790283, 1.304147720336914, -0.8009486198425293, -0.3981119394302368, -0.47328969836235046, 0.8560363054275513, -1.3302078247070312, -0.24449443817138672, 0.24302972853183746, -0.8040379285812378, -0.43502748012542725, -1.7941384315490723, -2.2676780223846436, -0.5248591303825378, -2.2460777759552, 1.1536489725112915, 0.9777312874794006, 0.895704984664917, 1.1410562992095947, -1.0467694997787476, 0.6442916989326477, 0.9090408086776733, -0.8802698254585266, -0.39484137296676636, 0.6346110701560974, 0.6317172646522522, 0.3492719829082489, 0.04043873772025108, 0.7859033346176147, -0.4677746593952179, -0.2383553385734558, -0.29360973834991455, -0.18720366060733795, -0.8094542622566223, 0.266442209482193, -1.2304117679595947, -0.5597527623176575, -0.1700892150402069, -1.8787809610366821, -1.6325442790985107, 1.3913443088531494, 0.7474070191383362, -0.38875001668930054, -0.7376176118850708, 2.869041681289673, -0.21288104355335236, -1.2944116592407227, -0.4955805242061615, -0.9761817455291748, 0.14631441235542297, -0.6065112948417664, -0.5315938591957092, 1.1230106353759766, 1.717869758605957, 1.4822907447814941, -1.6268157958984375, 0.1393560767173767, 1.56009042263031, 0.19742007553577423, -1.393086552619934, 0.17997437715530396, 0.16013728082180023, -0.30510056018829346, 0.9564533233642578, 0.5816745162010193, 0.05693981423974037, 0.548144519329071, -0.2838534116744995, -0.43496763706207275, 0.7839282155036926, 0.6381221413612366, -0.8077272176742554, -0.197737917304039, 1.4865951538085938, 0.7405163645744324, 0.28417545557022095, -1.5644561052322388, 0.9299423694610596, 1.2481629848480225, -0.028593868017196655, 0.7563892006874084, -0.06513108313083649, 0.6214699149131775, -1.7398011684417725, -2.0550408363342285, 2.0581657886505127, -0.883874237537384, -0.9342854619026184, -1.1051892042160034, 1.7909616231918335, -0.4559841454029083, 0.8748320937156677, -0.0052886544726789, 0.6077248454093933, 0.2873536944389343, -0.0013876330340281129, -0.3959125280380249, 0.7068111896514893, -1.2389209270477295, -0.21678119897842407, -0.7476178407669067, -0.8142989277839661, -0.7407503724098206, -1.5526114702224731, 0.4598342180252075, 0.48457738757133484, -0.8398502469062805, 0.08488715440034866, 1.359475016593933, -0.36724671721458435, -0.01774846389889717, 0.01996106468141079, 0.5724767446517944, 1.7254140377044678, 1.9559017419815063, -0.5560309886932373, 0.9837389588356018, 0.46238306164741516, 0.6523049473762512, -0.12998738884925842, 0.7136996388435364, 0.08651220798492432, -0.07375805824995041, -0.6944436430931091, -0.10920655727386475, -0.17836038768291473, -0.9030404090881348, -1.003261923789978, -0.4952448606491089, 0.9005503058433533, -1.8172063827514648, -0.7461199164390564, -1.2135934829711914, -0.05883285775780678, 0.7001588940620422, 0.21344810724258423, -0.7833275198936462, -1.1576627492904663, -0.04534255713224411, -0.47760769724845886, 0.553139865398407, -1.9134560823440552, 0.23047085106372833, -0.9850225448608398, 0.7342936992645264, -1.2205997705459595, 1.4607924222946167, -0.326414555311203, -0.1245700791478157, 1.2465146780014038, -0.5140820145606995, -0.171646386384964, -2.4986138343811035, 1.192335844039917, -0.3752627670764923, 0.5511817336082458, -0.7852855324745178, -0.19446058571338654, 0.5154479146003723, 0.7263133525848389, -0.13370928168296814, 0.739288330078125, -1.9104607105255127, -1.005741000175476, 0.04097355529665947, 0.9783971905708313, 1.4595006704330444, 2.0247435569763184, -0.14087159931659698, -0.9888862371444702, 1.1699225902557373, 1.9386932849884033, -0.891437828540802, 0.2707575559616089, 1.0916767120361328, -0.17514732480049133, -1.1024386882781982, 0.23145882785320282, -1.1462297439575195, -0.48718029260635376, -1.301196575164795, 0.5797109007835388, 1.687968134880066, 1.0850507020950317, -0.41610050201416016, -0.5609273314476013, 1.612899899482727, 1.3104339838027954, -1.4739277362823486, 0.8126669526100159, -0.14452120661735535, 0.554284393787384, -0.5400394797325134, -0.23946239054203033, -1.230055570602417, -0.275762140750885, -0.49510350823402405, 1.042642593383789, -0.2304421365261078, -1.2623896598815918, -0.2117655724287033, -0.6769622564315796, 0.38225606083869934, 1.9577445983886719, -1.472316861152649, 1.7284324169158936, 1.2527252435684204, -1.2743167877197266, -0.44940000772476196, -0.04957851395010948, -0.4163920283317566, 0.08030078560113907, 0.5120164155960083, -0.8874989748001099, 1.4987709522247314, -1.1074440479278564, -1.7609798908233643, -1.4092267751693726, -0.16338053345680237, -0.3953428566455841, 0.286784291267395, -1.3777481317520142, -0.09938561916351318, -0.1961587369441986, -1.6483373641967773, -0.11575055867433548, -1.4326869249343872, 1.3528861999511719, -0.013241874985396862, 0.1010425016283989, -1.5823453664779663, -0.4503875970840454, 0.33291640877723694, -0.12478546053171158, -0.1864866018295288, -1.3038253784179688, -0.18714463710784912, -0.8901379108428955, -1.1991406679153442, -1.4448554515838623, 1.0474011898040771, 2.3079121112823486, 0.08381998538970947, 0.8881644606590271, -1.5336931943893433, -0.18959815800189972, -0.8861588835716248, 1.0325703620910645, -1.8160871267318726, -0.5322383642196655, 0.3052354156970978, -0.9702222943305969, 0.29219141602516174, -0.41132885217666626, -1.1047905683517456, 0.06738622486591339, -1.107407569885254, -1.2321984767913818, -0.6168672442436218, 1.3030086755752563, -1.376217007637024, 0.8563352823257446, -0.08851358294487, 0.8220365047454834, -0.2807738184928894, 1.2118028402328491, -0.6424582004547119, 0.5544623732566833, -0.7219881415367126, 0.7633593678474426, 0.3007534444332123, 0.4410739243030548, -0.21774478256702423, -0.7161188125610352, 0.30359429121017456, -1.7828147411346436, -1.0714062452316284, 0.8498698472976685, -0.3876292109489441, -1.1633830070495605, -1.3327194452285767, 1.7441489696502686, -1.067049503326416, 0.9157974720001221, -0.923828661441803, -0.09998191893100739, -1.6886459589004517, -1.3555458784103394, -0.8785944581031799, -0.5235357880592346, 0.24262294173240662, -0.28039687871932983, -1.170725703239441, 0.111107237637043, -1.0994558334350586, -2.2685604095458984, 1.7404690980911255, -0.7237868309020996, 0.8737518191337585, -0.5239726305007935, -1.0231027603149414, -1.220758318901062, -0.7064300179481506, -0.6954562067985535, 0.9670504927635193, -0.45392289757728577, -0.6271616816520691, -0.3433571755886078, 0.7672942280769348, -0.22473956644535065, 0.47502651810646057, -0.07878097891807556, 1.0428191423416138, 0.3315922021865845, -1.2163267135620117, 0.7164896726608276, 1.67097806930542, 0.12093964964151382, -0.6247673034667969, -0.8155136704444885, 2.623439073562622, -0.7850458025932312, 2.3999102115631104, -1.1204524040222168, 0.010260799899697304, 0.5607372522354126, -0.38175079226493835, 0.6207970976829529, -0.7802561521530151, 0.6113722324371338, 0.028227418661117554, -1.1462578773498535, 0.013619087636470795, 0.2039116472005844, -2.428274631500244, 1.1875039339065552, -0.5101752281188965, -1.3576289415359497, 1.3925623893737793, -1.67990243434906, -0.1531064212322235, -0.29870539903640747, 1.0645232200622559, 1.3131780624389648, 1.175358772277832, 0.023158635944128036, -1.188865303993225, -2.037160873413086, 0.20509961247444153, 0.4622940123081207, -0.7155694365501404, 0.09415113180875778, 1.1336376667022705, -0.8307754397392273, 0.6324658393859863, 1.3219499588012695, 1.3867297172546387, -0.16319780051708221, 0.36863452196121216, 0.37891992926597595, 1.4765607118606567, 0.8931679129600525, 0.32607829570770264, 2.463146924972534, -0.519305408000946, 0.21288534998893738, -1.7662854194641113, -0.9207978248596191, 1.731281042098999, 1.468570590019226, -1.2920812368392944, 0.577373206615448, -0.24786733090877533, 0.31355735659599304, -0.08800103515386581, 0.7180066704750061, 1.5202232599258423, -0.7417123317718506, 0.3128070533275604, -1.2332395315170288, -0.8190009593963623, -0.30008870363235474, -0.8502662777900696, 0.8052330613136292, 0.07845257967710495, -2.6431872844696045, -0.6064709424972534, 0.8671140670776367, -0.09283795207738876, -0.14404453337192535, 1.3256956338882446, 1.7896246910095215, 1.7860674858093262, 0.351978600025177, -0.2932404577732086, -1.7141153812408447, 0.8858956098556519, -0.6398628950119019, 0.49325430393218994, 0.28467535972595215, 0.00029832826112397015, -1.939400315284729, 1.2901439666748047, 0.21674758195877075, -1.4716233015060425, -1.17947518825531, -0.005582272540777922, -0.004939820151776075, 0.470615029335022, -1.257562279701233, 0.8052463531494141, -0.2675061821937561, 0.2514343857765198, 0.46508821845054626, -0.6209595799446106, 0.46691828966140747, -0.22848562896251678, -0.1496424525976181, -1.6304172277450562, -0.6705322265625, -0.12131153792142868, 0.5239039659500122, -0.12848812341690063, -0.8793138861656189, -0.3711158037185669, 0.5291842222213745, -0.10142336040735245, 0.2692107558250427, -1.7208802700042725, -1.5877128839492798, 1.3972218036651611, 1.47187340259552, -1.4238585233688354, 0.3919648230075836, 0.6761291027069092, -0.36772647500038147, 0.8805627822875977, -0.09316423535346985, -0.02919125184416771, -0.9566996693611145, -0.06693409383296967, 0.3679254949092865, -1.2878252267837524, 1.1729940176010132, -0.522076427936554, -0.9042583107948303, 0.222072571516037, -0.903856098651886, 1.0714449882507324, -1.1769814491271973, 0.038658492267131805, -1.1774441003799438, 0.20064043998718262, -0.014103114604949951, -0.8830025792121887, -1.2210724353790283, 0.1504351943731308, 0.3816685080528259, 0.27368614077568054, 0.1944011002779007, 0.8701874613761902, -0.7553192377090454, 1.7297558784484863, -0.8944068551063538, 1.554304599761963, -0.48423701524734497, -1.3117517232894897, -0.8625434041023254, 0.5576778650283813, 0.6663632392883301, 0.35052576661109924, 1.1044566631317139, 0.5622657537460327, -0.028718968853354454, 1.3127772808074951, -0.11420589685440063, 0.9329899549484253, 2.9134130477905273, -0.39234110713005066, 0.4327421188354492, 0.5043470859527588, 0.887754499912262, -1.38113534450531, 0.5458080768585205, 0.026544731110334396, 1.3176496028900146, -0.06400583684444427, -0.428205668926239, 0.3627099096775055, -0.037112459540367126, 0.13840846717357635, 0.11606335639953613, 0.1217532753944397, 1.0857173204421997, -0.7329534888267517, -0.9785370826721191, 1.631036400794983, 0.6734938025474548, -0.3471584916114807, 0.11817661672830582, 1.5607223510742188, -1.6698650121688843, 0.517583966255188, 0.13107195496559143, -0.3649134337902069, 0.7791491746902466, -0.28888505697250366, -0.3996082544326782, -1.7707879543304443, -0.394133597612381, 0.5429400205612183, 1.1711393594741821, -0.5152077078819275, 0.6700872778892517, -1.650132417678833, -1.3063265085220337, -0.3019232451915741, 1.0988984107971191, 0.3501598536968231, 0.6697904467582703, -0.37400317192077637, -0.6526747345924377, 1.0242516994476318, -0.007555011194199324, -0.11601462215185165, -0.129500150680542, -1.1485724449157715, 1.3341808319091797, 0.5799134373664856, 1.1073086261749268, -1.206526756286621, 0.6606022715568542, -1.0613605976104736, 2.226686477661133, -0.3378041684627533, 1.3166124820709229, -0.2328999787569046, 2.5921378135681152, 0.7942959070205688, -0.5357915163040161, -0.0066228401847183704, 2.2680349349975586, 1.0856674909591675, 0.09790554642677307, -0.8566712141036987, 1.267633318901062, -0.8160914778709412, 0.12583108246326447, 0.8791087865829468, -0.055065643042325974, 0.1418994516134262, 0.3099648952484131, -0.12664270401000977, 1.1182243824005127, 0.754684567451477, 0.38707998394966125, -0.8725050687789917, 0.7798963785171509, 0.0698239654302597, 0.17498312890529633, -0.5345890522003174, 2.0215461254119873, 1.3056349754333496, -1.450469970703125, 1.9276565313339233, 1.4771714210510254, -0.49969959259033203, -0.2965274155139923, -0.11966803669929504, 1.3431659936904907, 1.307039737701416, -0.7489341497421265, 0.05574920028448105, 1.321528673171997, 0.0944237932562828, 0.12864138185977936, -1.316422462463379, 0.2729662358760834, 1.1094598770141602, -0.66310715675354, 0.8655182719230652, 2.097966194152832, -0.6772542595863342, 0.24290665984153748, -1.1308584213256836, 0.2598649859428406, 1.360887885093689, -0.5952824950218201, -1.8068015575408936, 0.5644388198852539, -0.11320438235998154, 1.983974814414978, -2.017423391342163, 0.17637018859386444, 1.6220269203186035, -0.5480005741119385, -1.1972150802612305, -1.7194796800613403, -1.6782653331756592, -0.11207187920808792, -1.53761887550354, 0.3062813878059387, 0.27119091153144836, 0.12355148792266846, 0.4170842468738556, 0.7980377674102783, -0.11252371221780777, 0.4023115932941437, -0.20187610387802124, -0.8986329436302185, 0.9060235023498535, 0.2319406270980835, -0.95174241065979, 0.9136132001876831, -0.4880308508872986, -0.26123836636543274, -0.02096758410334587, 0.6760297417640686, -0.27705952525138855, -1.5247020721435547, 0.2555619478225708, -0.5088119506835938, -1.2849594354629517, 0.2314993441104889, 0.9277426600456238, -0.24453529715538025, 0.6718717813491821, -0.38440942764282227, -0.22395046055316925, -0.04368729889392853, 0.030048992484807968, 0.8684437870979309, 1.09829843044281, -1.3461191654205322, -0.6012021899223328, 1.4720971584320068, -0.6170421242713928, 2.073184013366699, -0.06424883008003235, -1.2121517658233643, -1.0722898244857788, 0.706906259059906, 1.0931880474090576, -1.79343581199646, -1.297702670097351, -0.32295703887939453, 1.9898797273635864, 0.7338505387306213, -0.6649846434593201, -1.010438323020935, -0.31867119669914246, -0.4252237379550934, -0.695244550704956, 0.4488106071949005, 0.3943604528903961, 2.142725706100464, 0.6013766527175903, -0.38082996010780334, 1.0072567462921143, 1.5343552827835083, 0.9084174036979675, -0.06967868655920029, -0.16111139953136444, -1.2389711141586304, 1.7310395240783691, -0.8326320052146912, -1.0186405181884766, 0.646945595741272, -0.8425637483596802, 0.34218090772628784, 0.49331188201904297, -0.8232623934745789, -2.01650333404541, 2.0643346309661865, -0.6494736075401306, -0.5079603791236877, -1.4638426303863525, -1.6850895881652832, 0.31281882524490356, -1.0053719282150269, 1.2130985260009766, 0.5681076049804688, 0.31600573658943176, 0.5708093047142029, 1.5498669147491455, 0.570473849773407, -0.728318452835083, -1.7873119115829468, -0.19770456850528717, 0.3235047161579132, 1.7222408056259155, 0.5457863211631775, 2.171523332595825, -0.7454191446304321, -0.6647807955741882, 1.2572693824768066, -0.9878070950508118, 0.31937944889068604, -0.24252036213874817, -1.974331021308899, -0.5332710146903992, 1.7828025817871094, 1.028292179107666, 0.4272845983505249, -0.9677191376686096, -0.532406747341156, -1.7322269678115845, 0.44008108973503113, 0.21046340465545654, 0.2796929180622101, 1.08302903175354, 1.5595152378082275, -0.221971794962883, -0.6186308860778809, 0.8495084047317505, 0.5339127779006958, -0.6310017704963684, -0.0777377113699913, -1.6905194520950317, -0.2154891937971115, -0.32699793577194214, -0.909142255783081, 0.09379425644874573, 0.9391411542892456, 0.7258797287940979, -0.005365666002035141, 0.3194582462310791, -0.606302797794342, 1.9822763204574585, -0.4747503697872162, -2.209542751312256, 1.880393624305725, -0.6346760988235474, -1.2054505348205566, -0.5197891592979431, 0.10105764865875244, -1.1727714538574219, 1.4797223806381226, -1.1998339891433716, 0.372683584690094, 0.9576379060745239, -0.24799180030822754, -0.843350887298584, 0.7359189391136169, 0.05100728198885918, -0.4759661853313446, -1.0999841690063477, 0.03448617085814476, -0.3694353401660919, -0.8316961526870728, 0.13999059796333313, -1.0836920738220215, -1.5769914388656616, 0.016715265810489655, -0.4929966330528259, 1.8909354209899902, -0.43658891320228577, -1.2001234292984009, -1.0009503364562988, 0.5864623188972473, 0.02276829071342945, 0.29468968510627747, -0.9582712054252625, -0.8974570631980896, -1.4187589883804321, 0.37503889203071594, 0.6789547204971313, 0.4479590952396393, -0.5065333247184753, 0.8833522200584412, 1.8650895357131958, -0.5458051562309265, 1.2231308221817017, 1.8549718856811523, -1.3602372407913208, -0.29207879304885864, -0.7777737975120544, 0.9508770108222961, -0.5267135500907898, -1.589627981185913, 1.8625394105911255, 0.3243962228298187, -0.3922313451766968, 0.336076945066452, -0.5174263119697571, 1.73979651927948, -0.25413602590560913, 0.9592505693435669, 0.2604658305644989, -0.6435277462005615, -0.39083418250083923, -0.28592851758003235, -0.33209216594696045, 0.42950454354286194, 0.12454665452241898, -0.7652308344841003, 1.0448274612426758, 0.15348182618618011, 0.5624134540557861, -1.1117459535598755], [1.4895135164260864, 0.9892870783805847, 0.3597538471221924, -0.7489027976989746, 1.0499643087387085, 0.879788339138031, 0.18560248613357544, 0.5118886232376099, 1.4160571098327637, -1.3567701578140259, -1.1723711490631104, -0.5551702976226807, -1.2029435634613037, -0.39029228687286377, 0.2371201068162918, 3.094515800476074, 0.20641137659549713, -0.27620524168014526, 0.17906881868839264, 0.13265444338321686, 1.6643977165222168, 0.6846247315406799, -0.5849338173866272, 0.2612815797328949, -1.0456507205963135, -1.3644118309020996, 0.2513132393360138, 0.8305364847183228, -0.64938884973526, 0.05929352343082428, -0.6153725385665894, -0.2656344771385193, 0.1172005832195282, 0.21561512351036072, 0.6245729923248291, 0.575072169303894, -1.0324139595031738, 0.05989968776702881, 1.4967442750930786, 0.31298667192459106, 1.4392976760864258, 0.751769483089447, -0.7834999561309814, -0.2624979615211487, 0.6945990324020386, -0.16777415573596954, 0.9226676821708679, 1.2914503812789917, 0.31590983271598816, 1.0135427713394165, 0.6511889100074768, -0.09472966939210892, -0.5883447527885437, 1.9153289794921875, 0.7086521983146667, -0.3736741542816162, 0.5389421582221985, -0.3841705024242401, -0.10713072121143341, 1.8887370824813843, -0.5714455246925354, 2.852653980255127, -1.1859205961227417, -0.03694131225347519, -0.027042798697948456, -0.027207542210817337, 1.4765737056732178, -1.3021811246871948, 2.2704949378967285, -0.25654852390289307, 0.801734447479248, -0.12015970796346664, -2.4235825538635254, -0.6565747261047363, 1.420257568359375, -0.5649991631507874, -1.2150098085403442, -0.30999940633773804, -0.014768910594284534, 0.6929320096969604, 0.8587315082550049, -0.8876312375068665, -0.5767176151275635, 1.2559540271759033, -0.2778981328010559, 0.31810495257377625, 0.12852180004119873, -1.4550514221191406, 0.7174659967422485, -0.6861087679862976, 1.785504698753357, 1.1602388620376587, -1.0060597658157349, 0.5914101600646973, 0.9809924960136414, 0.38613584637641907, -1.5347734689712524, -1.033632755279541, -0.2415519505739212, -1.0247578620910645, 2.563443660736084, -1.3019407987594604, -0.4711078703403473, 0.3879324495792389, 0.7218642830848694, 2.3879776000976562, -1.1285974979400635, -1.3720961809158325, 1.0728992223739624, -0.8128412365913391, 0.44537827372550964, -0.8173529505729675, 0.3527604341506958, 0.5096194744110107, -0.19189117848873138, 0.5049226880073547, -0.7703725099563599, -0.5049970746040344, -0.5317371487617493, -1.0692813396453857, -0.42135196924209595, -1.2187877893447876, -0.8078477382659912, 0.2667751908302307, 0.8462031483650208, 0.4719228148460388, -0.9621041417121887, -0.5665632486343384, -0.67519211769104, -0.09428152441978455, -0.17341428995132446, 0.05623449757695198, 1.325196385383606, -0.25673407316207886, 0.3176578879356384, -0.1829172521829605, 0.6680696606636047, -0.5803336501121521, 0.8632090091705322, 1.0197010040283203, 1.1540528535842896, 0.3699280917644501, -0.7513350248336792, 0.7374646663665771, -0.15935778617858887, 0.07275232672691345, 1.0432147979736328, -0.5339057445526123, 0.10525915771722794, -0.3246905207633972, 2.1685550212860107, -0.9606516361236572, 1.9201984405517578, 0.16813087463378906, 0.6361023783683777, -0.29973775148391724, -1.2672301530838013, -0.3069475591182709, -0.49037665128707886, -0.8877525925636292, -1.6288753747940063, -0.6930373907089233, 2.4934256076812744, -1.8859879970550537, 0.3759591579437256, 1.4989222288131714, 0.2574447989463806, -1.5312215089797974, 1.5856420993804932, -0.2739376425743103, -1.4308162927627563, 1.6858150959014893, 1.920885682106018, 0.8721892237663269, 0.6386772990226746, -1.3898309469223022, 0.2291393131017685, -1.1992462873458862, 0.2059876173734665, 0.5135863423347473, -1.197564721107483, -1.1351300477981567, 1.6211918592453003, 0.47784510254859924, 0.1444837599992752, -0.7313059568405151, -0.25022244453430176, -0.8737401366233826, 0.9443665146827698, 0.17109620571136475, -1.5002259016036987, -1.2853485345840454, 0.10289915651082993, -1.329095721244812, 0.25783196091651917, -1.8172166347503662, 0.9837791323661804, 0.6585855484008789, -1.2339061498641968, 0.6234268546104431, -1.2318562269210815, -1.7818365097045898, -0.6222783327102661, 0.4113973081111908, 0.38806331157684326, -0.6366423964500427, 0.813389241695404, 1.1646093130111694, 0.3768985867500305, 0.05996118113398552, -0.3294084966182709, -0.6901540756225586, -3.204151153564453, 0.9208800196647644, 0.6343802213668823, -0.6831142902374268, 0.23625829815864563, 0.5313332676887512, -0.9307160377502441, 2.0597622394561768, -2.0479278564453125, 1.087733268737793, -0.5043998956680298, -1.6040968894958496, 0.22707012295722961, 0.32954567670822144, -0.7769577503204346, 0.5964874625205994, 1.9427382946014404, 1.7739994525909424, 3.077183246612549, 0.34645533561706543, -0.43884381651878357, -0.4674226641654968, -1.3052855730056763, -0.43327850103378296, 1.121878743171692, 1.335853934288025, -0.7380317449569702, 0.006392878945916891, 0.05576734244823456, 1.6459468603134155, -0.41251853108406067, 0.7794824242591858, 2.2047948837280273, -1.817849040031433, 0.9507095813751221, -1.655835509300232, 0.8614847660064697, -2.7330780029296875, 0.5534548759460449, -0.6363434195518494, -0.8760877847671509, -0.2089461237192154, -0.6213812232017517, 1.519982099533081, -0.47646304965019226, 1.5192078351974487, -1.3619534969329834, 0.599092960357666, 0.7121274471282959, -0.9500876069068909, 0.9193137288093567, 0.17403818666934967, -0.32674720883369446, -1.0988061428070068, -0.687256395816803, 1.3607463836669922, 0.9878966808319092, -0.1644441783428192, 0.47498464584350586, 0.2925650477409363, -0.8730534315109253, -1.8497061729431152, 1.3102521896362305, 0.21024975180625916, -0.10606547445058823, -0.4393615424633026, 1.9405238628387451, 1.2422195672988892, -1.2524104118347168, 0.9125633835792542, -0.5148650407791138, 0.41852498054504395, -2.4831490516662598, 1.4780089855194092, 0.1191897988319397, 0.1687721461057663, -1.219128966331482, -0.8828600645065308, 1.212988257408142, -0.29640093445777893, 2.7876107692718506, 0.5737248063087463, 0.3789544403553009, 0.7995823621749878, -0.7085129618644714, -0.6643951535224915, -2.907365560531616, -0.8367419242858887, 1.0330679416656494, -0.027337579056620598, 1.7190964221954346, 0.3408482074737549, 0.81425541639328, 0.5696688890457153, 1.0592858791351318, 0.21043604612350464, 0.9770033955574036, -0.44750484824180603, -0.5837150812149048, -0.7695916891098022, -0.055096082389354706, -0.4885903298854828, 0.029311465099453926, 0.4105879068374634, -0.4900064766407013, 0.7666252851486206, 0.5895749926567078, -1.6261767148971558, 0.746514081954956, 0.4649313986301422, 0.4081510603427887, -0.46749818325042725, 0.1630028635263443, -1.0144504308700562, 0.8903366327285767, -0.7556294798851013, -1.705965280532837, -1.0671584606170654, -0.5537713766098022, 0.8108274340629578, 0.3825710713863373, -1.5608882904052734, -0.717097818851471, 0.3728223145008087, -0.14254818856716156, 1.0695579051971436, 0.8584845066070557, -0.5409377813339233, -0.2314741015434265, -0.020306233316659927, -1.1331186294555664, 1.3789857625961304, 0.6556610465049744, -0.39213308691978455, 1.472255825996399, 1.3026597499847412, -0.019318269565701485, 1.2668415307998657, 0.4197464883327484, -0.186653271317482, -2.1065831184387207, 0.33936798572540283, -0.33174994587898254, 0.593911349773407, 0.017978094518184662, 1.341048002243042, 0.657347559928894, -1.0916537046432495, -1.0214653015136719, 0.39805567264556885, -1.4565929174423218, -0.03607718646526337, -1.9882607460021973, -2.773670196533203, -1.3951712846755981, 1.2530031204223633, 1.072049617767334, 0.23596403002738953, -0.33285483717918396, -1.2143616676330566, 1.2649329900741577, -0.8610872626304626, 0.40717214345932007, 1.3422255516052246, -0.74269700050354, -0.615459144115448, 0.5832244157791138, -1.718027114868164, -0.5655502676963806, -0.37821707129478455, -1.8248775005340576, 1.126804232597351, 1.6582962274551392, 0.04484580084681511, -0.9897821545600891, 0.20955291390419006, 0.5431450009346008, -0.17586562037467957, -0.702261209487915, 0.6117540597915649, -0.39575859904289246, -1.4076743125915527, 0.31015101075172424, -0.12433508783578873, -0.060189325362443924, 0.2874687910079956, 0.5824084877967834, 2.0014069080352783, 0.3008756637573242, 1.1582804918289185, 0.9266099333763123, 0.004286948591470718, -0.646030843257904, 1.9619866609573364, -0.6459414958953857, 0.7738445997238159, -1.3107327222824097, 0.23006299138069153, 1.8953487873077393, -1.041386604309082, 0.4928329586982727, -1.4015411138534546, -0.20837202668190002, -2.0659945011138916, -0.944664478302002, 0.7910512685775757, 0.2120475471019745, 1.1964541673660278, 0.9687332510948181, 0.012017199769616127, 1.6754724979400635, -1.2862468957901, 0.069658562541008, -0.005607293453067541, -0.5918343663215637, 0.42107170820236206, -1.6329894065856934, -0.8561644554138184, -1.3562545776367188, 0.40432044863700867, 0.5164251923561096, 0.9104416370391846, 1.6615701913833618, 1.5595746040344238, 0.8571141362190247, -0.8694849014282227, -0.13303986191749573, 1.224043369293213, -0.06276162713766098, 0.10813823342323303, 0.5881822109222412, 0.7713189721107483, 1.1907110214233398, -0.05068088322877884, 1.608734369277954, -0.2558843791484833, -0.40193381905555725, -0.19681698083877563, -0.9427469372749329, 0.38091087341308594, 1.206858515739441, -0.8506436347961426, 2.6203956604003906, -1.0275366306304932, -0.19940654933452606, 1.6518256664276123, 0.011506603099405766, -1.9405924081802368, 0.3506627380847931, -0.0636659786105156, -1.3077043294906616, 1.6537678241729736, 0.16031840443611145, 0.2443339079618454, -0.5191138982772827, 0.42114126682281494, -2.0876591205596924, 0.3157823383808136, 0.3715790808200836, -0.18075020611286163, -2.056259870529175, 0.6095033884048462, 0.28054043650627136, 0.5363509058952332, -1.025638461112976, -0.5113120079040527, -0.7152258157730103, -1.3606877326965332, -1.0321815013885498, -0.43123701214790344, 1.462994933128357, -2.5667636394500732, -0.28557008504867554, -1.3605682849884033, 2.0337307453155518, 1.5563160181045532, 0.0775245875120163, -1.0261812210083008, -1.37477707862854, 0.3212539255619049, -1.7087862491607666, -0.28406575322151184, 1.2272772789001465, 0.00538775697350502, -1.2105002403259277, 0.22914829850196838, -0.19406309723854065, 2.081653356552124, 1.2588003873825073, 0.17333345115184784, -0.1560370922088623, 1.1371647119522095, 0.07140011340379715, -0.7155317068099976, -0.9913209676742554, -1.783639907836914, -0.856139063835144, 1.4729450941085815, -0.43761399388313293, 0.3656021058559418, -0.5403253436088562, -1.266072392463684, 0.3132498264312744, 1.3713195323944092, 0.06553400307893753, -0.5851012468338013, -0.21668775379657745, -1.8773120641708374, 0.585649847984314, -1.165789008140564, -1.249538540840149, -0.6566861271858215, 0.4904164671897888, 0.07636940479278564, 0.03757916018366814, 1.47737455368042, 1.2600644826889038, -1.89885413646698, -1.5708179473876953, 0.9346857070922852, 1.2382354736328125, 0.4619106948375702, -0.6394715905189514, 2.7168986797332764, 0.010404649190604687, -1.0606743097305298, -2.130108118057251, 0.6906036734580994, -0.03728780895471573, 0.08009926974773407, -1.4616727828979492, -0.878215491771698, -0.1286795288324356, 0.9818814992904663, -0.8466426730155945, 0.6407076120376587, -0.07152511179447174, 0.5660476684570312, 0.026648685336112976, -2.1235764026641846, -0.7858008146286011, -0.6791403889656067, -1.1474272012710571, 0.6718357801437378, -1.0419012308120728, -0.6189931631088257, 1.6743559837341309, -0.1092747375369072, -0.31456756591796875, -0.3917566239833832, 0.36805590987205505, -0.023544874042272568, 1.3751881122589111, 0.7511200308799744, 0.1101635992527008, -0.13920949399471283, 0.5064891576766968, 1.3223869800567627, -0.01736704632639885, 0.9157243371009827, 1.6620997190475464, 0.47902533411979675, -1.0206353664398193, -2.195385456085205, -0.40210995078086853, -0.47493526339530945, -0.8219757080078125, 1.3176411390304565, 0.7447935938835144, -1.1113526821136475, -0.6622865200042725, 0.09875456243753433, -1.0773849487304688, 0.6799988746643066, -0.2911083996295929, -1.1513103246688843, -0.5052763819694519, 0.33701813220977783, -0.35000845789909363, -0.775557816028595, 0.47884324193000793, 0.8721277117729187, -0.510111927986145, 1.189487099647522, 0.269651859998703, -0.8293703198432922, 0.7926174402236938, -1.5385626554489136, 0.37663280963897705, 0.2582522928714752, 1.461371898651123, -1.3935697078704834, -0.043405063450336456, 0.38759472966194153, 0.8076250553131104, -0.027994850650429726, -0.2196323573589325, 1.0196503400802612, -0.5966288447380066, 1.867477536201477, -0.23549886047840118, 0.08608170598745346, 0.34308913350105286, -0.3318091630935669, 0.4687395691871643, 1.3559625148773193, -0.8774885535240173, -1.104774832725525, 0.2058882862329483, -0.22179481387138367, -0.4161604642868042, -0.9874865412712097, -1.4760119915008545, -0.0771939679980278, 2.007488965988159, 1.3850358724594116, -0.632473349571228, -0.8812909126281738, 1.9665718078613281, -1.4210478067398071, 0.01294984482228756, 0.3802480697631836, 1.6505298614501953, 1.0596349239349365, 0.04094680771231651, 0.01136445626616478, -0.5708490014076233, -1.009036898612976, 1.3587976694107056, 1.2213670015335083, 1.386237621307373, -0.7525907754898071, 1.5931386947631836, 0.09539154171943665, 0.13983356952667236, -0.3576414883136749, 0.3121509850025177, 2.6412057876586914, -0.14104141294956207, -0.3544646203517914, -0.6370368599891663, 2.0760996341705322, -0.42825841903686523, 0.7242328524589539, 0.8198517560958862, 0.8767315745353699, -1.346218228340149, 1.0229026079177856, 0.4545629024505615, 1.5027996301651, -0.8760303258895874, -0.6158351302146912, 0.41723939776420593, 0.4535847008228302, 0.3913009762763977, 1.3884727954864502, -2.2270781993865967, -0.9154098629951477, 0.6950833201408386, -0.21626560389995575, -0.08687659353017807, -0.5997475981712341, 0.4987359046936035, -0.19812722504138947, -2.5671451091766357, -0.7632246613502502, 0.41340190172195435, 0.07818906009197235, 0.5723456740379333, -0.22268179059028625, -0.47480952739715576, -1.225948691368103, 0.153425395488739, -0.527594268321991, 1.5385401248931885, -0.6040531396865845, -0.693797767162323, -0.48378244042396545, 2.1072824001312256, -1.6766252517700195, -0.4768252968788147, 1.280906319618225, -0.722679078578949, -0.8675923347473145, -2.2250728607177734, 0.6504295468330383, 1.0788567066192627, 1.2618920803070068, 0.6671070456504822, -1.0983575582504272, -1.0272133350372314, -1.080954670906067, -1.2063488960266113, 0.7085201740264893, -0.8012003898620605, -0.3728981912136078, -0.22736558318138123, -0.2089737504720688, -0.2833840847015381, 0.6279839277267456, 1.8583433628082275, 0.6508054733276367, 0.4472559094429016, 0.8053149580955505, -0.23385341465473175, -0.9465175867080688, 0.22290097177028656, 1.1841667890548706, -0.6030325889587402, 0.5538691878318787, 1.4305185079574585, 1.1774042844772339, 0.8686465620994568, -1.260712742805481, -0.41167980432510376, -0.5630042552947998, -1.190515398979187, 1.2172313928604126, 0.07469366490840912, -0.6989926099777222, -0.5525849461555481, -0.11539424955844879, -0.07610467821359634, 0.06400401890277863, -0.500176191329956, -0.6721224784851074, 0.37362775206565857, 0.7592014074325562, -2.40177059173584, -0.7867038249969482, -0.37240341305732727, 0.4292849600315094, -1.7440247535705566, -0.6649929285049438, 0.6127447485923767, -0.035194844007492065, 0.002916913013905287, -0.6513311862945557, 0.8207805752754211, -0.038708657026290894, -0.48202675580978394, -0.07002515345811844, -1.428963303565979, 0.5505704879760742, 0.7461918592453003, 1.352299690246582, -0.7586171627044678, 0.2940809726715088, -1.0556819438934326, 0.635995626449585, 0.015057362616062164, 0.07624835520982742, -2.1853950023651123, 2.501958131790161, 0.6316331624984741, -2.1283819675445557, 1.6694988012313843, -0.4402388334274292, 0.5671959519386292, -0.8056778907775879, -0.5853226184844971, -0.38907182216644287, 3.073887825012207, 0.32658469676971436, -1.5693632364273071, -1.3978402614593506, -1.008560061454773, 0.6268531680107117, -0.18281160295009613, -0.39232826232910156, 0.5704596042633057, 0.07070525735616684, -1.534087061882019, -0.989116370677948, -0.23524582386016846, 0.9139806628227234, -0.9158841967582703, 0.9130362868309021, -0.9027336835861206, 0.1650913506746292, -1.3136746883392334, -0.17430923879146576, 0.08781541883945465, -0.005936207715421915, -0.8246999382972717, -0.04259924590587616, -0.964179277420044, 1.429277777671814, 0.7890708446502686, 0.15538723766803741, -1.0733368396759033, -0.7891093492507935, -0.3252548575401306, 0.7767692804336548, 0.8672402501106262, -0.7126696705818176, 0.23139934241771698, 0.41894426941871643, -0.2326020449399948, 0.8074654340744019, 0.4600754678249359, -0.839758038520813, -0.5546332597732544, -0.8839408755302429, -0.6576756238937378, -0.870279848575592, 0.03647354245185852, -0.576643705368042, 0.3153502345085144, 1.9109197854995728, -0.2163907289505005, -1.007277488708496, -0.12669864296913147, -1.1606190204620361, -0.16063837707042694, -0.0013533104211091995, -0.8892256617546082, -0.9916287660598755, 1.3696706295013428, -0.4319753646850586, -0.5018633604049683, 0.483645498752594, -1.0767385959625244, -0.3908182978630066, 0.9600432515144348, 1.6464225053787231, 0.5964177846908569, 0.12889136373996735, -0.218665212392807, -0.3924906849861145, -0.8854531049728394, 0.6420711278915405, -1.0541737079620361, 0.9238532781600952, -2.637089252471924, -0.556423008441925, -0.03205125406384468, -0.7353861331939697, 0.08624791353940964, -0.5382937788963318, -0.5600216388702393, 0.22022277116775513, -1.0512553453445435, -1.8232253789901733, 0.5998632311820984, 0.32610562443733215, 0.15946444869041443, -0.9321857690811157, -1.0354875326156616, 0.7662165760993958, -0.9405338168144226, 1.6715649366378784, -0.8396586179733276, -0.48539143800735474, -0.6640239953994751, -0.05720146745443344, 2.361973762512207, 0.027620302513241768, -2.4293320178985596, 0.07880321890115738, -1.4725278615951538, -0.6244943141937256, 0.22776588797569275, -0.7067432403564453, 1.052220344543457, -1.92207932472229, 1.293835163116455, 0.6913843750953674, 0.03773736208677292, 0.7737560272216797, 0.13263331353664398, -2.0431225299835205, -0.4024665057659149, 0.21871088445186615, -0.0936274304986, 0.7754087448120117, -0.4901656210422516, -1.0781677961349487, -1.414808988571167, 0.08421696722507477, -0.04186149314045906, -0.3076312243938446, 0.04373558610677719, -0.9476255774497986, 0.5869296193122864, 1.0750741958618164, 0.47127318382263184, -1.5004394054412842, 0.6382951140403748, 0.40587809681892395, 0.3143293261528015, 0.24647988379001617, 0.3652704954147339, 0.02341006137430668, 0.9171686768531799, -0.7392892241477966, -0.27558863162994385, 0.08137223869562149, 2.4752044677734375, 0.9605922102928162, -0.2335842251777649, -1.058618187904358, 0.7301898002624512, -0.6056166291236877, 0.10205096006393433, -0.1705462485551834, -0.6055580973625183, 1.7665491104125977, 1.0948035717010498, -0.8620765805244446, 1.5667531490325928, -0.59809809923172, 0.17016227543354034, -1.7738300561904907, 0.5584890246391296, 0.9482787847518921, -0.442167192697525, -0.6402406692504883, -1.5270005464553833, -0.18653850257396698, -1.6358988285064697, 0.03675888478755951, -1.6430697441101074, 0.12990817427635193, 0.010890929028391838, -0.10712403804063797, 1.7365294694900513, -1.5072771310806274, -0.2795252501964569, -0.38030698895454407, -1.718321442604065, 0.15779373049736023, -0.5884206891059875, 0.6091480851173401, -1.058358907699585, -0.3423572778701782, -0.617350697517395, -1.3264024257659912, -0.2033238261938095, -0.020815858617424965, 1.7597551345825195, -0.20986470580101013, 1.21005380153656, 1.0645185708999634, 0.49287131428718567, -0.5979940295219421, 0.8227807879447937, 0.23982971906661987, -0.9635936617851257, 0.02506694570183754, -0.410388708114624, 0.36217379570007324, -2.0516910552978516, 0.34561771154403687, 0.6891555190086365, -0.39646992087364197, 0.6601126194000244, -0.133943110704422, 0.5220685005187988, -0.27804166078567505, -1.4454306364059448, -0.08116186410188675, -0.363574743270874, -0.175748810172081, 1.0585459470748901, -0.7835060954093933, -0.6828209757804871, 0.7755527496337891, -1.871931791305542, 1.5780775547027588, 0.8334231376647949, 0.23243282735347748, -0.4796201288700104, -1.2313495874404907, -0.8475149869918823, -0.7649081349372864, -1.3165334463119507, 1.3524407148361206, -1.2999894618988037, 1.283433437347412, -0.5764440298080444, 0.966995358467102, 0.6824740767478943, 0.577051043510437, 1.2167109251022339, -0.9967621564865112, 1.8436130285263062, 1.4816581010818481, -1.3960037231445312, -0.8819645643234253, -0.02960933931171894, -2.8610103130340576, 0.30114054679870605, 0.3185417950153351, -0.04186929762363434, 0.27008339762687683, -0.6898407936096191, -0.3710452914237976, 0.8012508153915405, 1.2658666372299194, 0.7047494649887085, -0.9664157032966614, -0.3980711102485657, -0.09900420904159546, 0.399704247713089, -0.7431169152259827, -0.9626736640930176, -1.4753402471542358, 1.043700933456421, 0.07650169730186462, 1.3514152765274048, 0.7398499250411987], [-1.4724136590957642, 1.121212363243103, 0.25789135694503784, -0.050531502813100815, 0.9360601305961609, 0.4496435225009918, -0.4950089454650879, 0.307664155960083, 1.6334497928619385, -0.5338295102119446, -1.1509333848953247, -0.9636887907981873, 0.47084861993789673, 0.925225555896759, -1.569758415222168, 0.3410344421863556, 1.0373749732971191, 0.8635495901107788, 0.5624208450317383, -0.9955579042434692, 0.13401982188224792, 0.6113370656967163, -0.08197849988937378, 0.7241412997245789, -0.5213146805763245, -1.2739118337631226, -1.1075313091278076, -0.08965770900249481, 0.7123326659202576, 0.17048397660255432, 0.9811734557151794, 0.8113590478897095, 0.6523224115371704, -1.855709433555603, 0.4723146855831146, -0.9453098773956299, 0.7144132852554321, -0.6241945624351501, -0.5975015163421631, -0.3231879770755768, -0.0972910225391388, -0.34183841943740845, 0.2147272527217865, -0.03216973692178726, -0.5962972044944763, -2.022329092025757, -0.22893834114074707, 0.28314217925071716, 0.8824297785758972, 1.8135125637054443, -1.2969380617141724, 0.3224732577800751, -0.2889671325683594, -0.2586279511451721, -0.28702089190483093, -0.2527886629104614, 0.20158934593200684, 1.5842094421386719, 0.37187692523002625, -0.006058746483176947, -0.8350309729576111, 0.8821967244148254, -1.1240953207015991, 1.1043415069580078, -0.04081998020410538, -0.6782118678092957, -2.0769083499908447, -1.5707370042800903, 0.32006365060806274, 0.03220634162425995, 1.0263944864273071, -0.8015296459197998, 0.4209277927875519, -0.024799205362796783, 0.346571147441864, 0.733535885810852, 1.19654381275177, 1.754424810409546, -0.3077978789806366, -1.3013235330581665, 0.33761054277420044, -0.9590935707092285, -0.15978190302848816, -0.886867344379425, -0.7219065427780151, -0.868906557559967, -0.19799204170703888, -1.2890269756317139, -1.9756585359573364, 0.8290600180625916, 0.6903371214866638, -0.9322369694709778, -0.5680946707725525, 0.14459453523159027, 0.3263660967350006, 1.304984211921692, -0.8591081500053406, 0.2829335629940033, -0.37217646837234497, 0.3295848071575165, 0.1659245491027832, 0.8860356211662292, -1.1650522947311401, -1.493944764137268, -0.5389678478240967, -0.6663494110107422, 0.5157450437545776, -0.9075260162353516, 0.5289857387542725, 0.42972099781036377, 0.5705346465110779, 1.2367721796035767, -0.6662973761558533, -1.2291145324707031, -1.648197054862976, -0.519929826259613, 2.029827356338501, -0.012402104213833809, 0.3201894462108612, 1.2285922765731812, 0.2353144735097885, 0.05952256917953491, 1.7750986814498901, -1.0454916954040527, 0.5048944354057312, -1.7247960567474365, -0.7977468371391296, -0.7391374707221985, -1.2959707975387573, 1.6716210842132568, 0.787422239780426, 1.5767748355865479, 1.3889169692993164, 1.1003406047821045, -0.5206141471862793, 0.4214861989021301, -1.7089364528656006, 1.4804400205612183, -0.816518247127533, 1.6397602558135986, 0.9102851748466492, -0.44426342844963074, 1.1826761960983276, 2.0961852073669434, 0.645614743232727, 1.7189512252807617, -0.45932185649871826, -1.1412620544433594, -0.010336427949368954, -1.5320740938186646, 1.14359712600708, 0.5586388111114502, -0.6960635185241699, 1.1011788845062256, 1.9186673164367676, 0.9629714488983154, -0.047493573278188705, -0.9088025093078613, 0.13803040981292725, 0.7762073278427124, -1.9553946256637573, -1.666385531425476, -0.7162933945655823, 0.675765335559845, -0.9205825924873352, -0.25387638807296753, 0.20083485543727875, -0.6837438344955444, 0.3528413772583008, 0.017211882397532463, 2.2436439990997314, 0.674382209777832, 0.6397231221199036, -1.4494556188583374, -0.0646946057677269, 0.3941696584224701, -0.3250441253185272, -1.3288133144378662, -0.8634902238845825, -0.1383891999721527, 1.2338037490844727, 0.7904819250106812, 0.4339655041694641, 0.37045660614967346, 0.75372314453125, -0.6513066291809082, 0.4905494749546051, -0.7190918922424316, -0.5513545870780945, -0.8119505643844604, -0.14708824455738068, 0.4326554238796234, 0.6317477822303772, -0.9693931341171265, -1.3717533349990845, -0.16349130868911743, -0.36509814858436584, 0.1697998046875, 1.0833375453948975, -0.7958492636680603, 0.9302231669425964, -0.6549485921859741, 0.022216279059648514, 0.6678013801574707, -1.8750838041305542, -0.809329092502594, 0.8998465538024902, -0.4510822296142578, -1.2266387939453125, -1.178409457206726, 0.8585113883018494, -0.7504145503044128, -1.6101139783859253, -0.3458874523639679, -0.5582383275032043, -1.5645849704742432, -0.015176709741353989, 1.024566411972046, -0.6134852766990662, 0.08193977922201157, -1.3022353649139404, -0.17172691226005554, -0.6786016225814819, -1.1733649969100952, 0.25549018383026123, 0.09512089192867279, 0.3032609522342682, 2.3527183532714844, -1.7259130477905273, 1.9602628946304321, 0.10275118052959442, -0.5210705399513245, 0.43385669589042664, -0.3554037809371948, -0.4380970299243927, 1.6948559284210205, 1.2910897731781006, 1.3620882034301758, 0.11461006850004196, 1.5064268112182617, 0.6378434300422668, 0.8804677128791809, -0.1606258749961853, 0.44059765338897705, -1.3382847309112549, -1.8404163122177124, -1.672806739807129, -3.18998646736145, 0.27300894260406494, 0.16238097846508026, -0.2686348259449005, 0.6901972889900208, 0.7584652304649353, 0.4948226511478424, 0.20805728435516357, -0.7820592522621155, 0.052090950310230255, 0.5032943487167358, -0.3513982594013214, -0.20884756743907928, -1.4778887033462524, -0.9392770528793335, 0.613896906375885, -0.8969588875770569, 1.5162229537963867, -1.364522099494934, -0.9599965214729309, 2.3605833053588867, -0.674051821231842, 0.06196870654821396, -0.6442942023277283, 1.0080279111862183, -1.3814010620117188, 0.44838812947273254, 1.8419822454452515, -0.7437365651130676, -1.8148267269134521, -0.5088150501251221, 0.7519438862800598, 1.8342869281768799, 0.006557023618370295, -0.25074493885040283, -1.1785094738006592, 0.3977900445461273, -0.21203628182411194, 0.8121879696846008, -1.159292221069336, -1.2255011796951294, -0.2366584688425064, -0.29126834869384766, -0.14763565361499786, -0.7107630372047424, 0.4531220495700836, 1.404218316078186, -1.0076484680175781, -0.024592110887169838, -1.5331062078475952, -1.3006529808044434, -0.4556899666786194, 0.21383097767829895, -0.282314270734787, -0.9218623638153076, 1.0874741077423096, -1.6734249591827393, 0.011064847931265831, 0.6454859375953674, -0.15256048738956451, 0.2989163398742676, -0.7178909778594971, -1.508467197418213, -1.627004623413086, 1.4042080640792847, 0.3786008954048157, 0.22810004651546478, -0.7558161616325378, -1.3854031562805176, 0.9487345218658447, 0.7982150912284851, -0.07233713567256927, -0.30129262804985046, 2.155489444732666, -0.8416759967803955, 0.8477943539619446, -0.05449647456407547, 1.807361125946045, -0.22519981861114502, 0.5654411911964417, 0.02494199573993683, -0.48532935976982117, 0.47337380051612854, -2.4635848999023438, -0.4203404188156128, 0.4604162871837616, -2.290541887283325, -0.710397481918335, -1.8338701725006104, -0.5658913850784302, -0.27279549837112427, 0.8406663537025452, 0.5107023119926453, -1.179760456085205, 0.6751978993415833, 1.4465633630752563, -0.3517349660396576, 0.16690798103809357, 0.9619276523590088, -0.35761234164237976, -0.3035912811756134, 0.10511810332536697, 0.4414820373058319, -0.09161560982465744, -0.27162113785743713, 0.1655520349740982, -0.08638115972280502, -0.47136473655700684, 0.9907110333442688, -1.1468530893325806, -0.08075980097055435, 1.1548274755477905, 0.1349039077758789, -1.1552236080169678, 0.32488521933555603, 0.8642902970314026, 0.9742025136947632, -0.5975974202156067, 0.5539208650588989, 0.44179150462150574, 1.3311846256256104, -0.510965883731842, -0.5610798597335815, 0.8738250732421875, 1.3294843435287476, -0.01439269445836544, 1.3930156230926514, -1.236545205116272, 2.2682909965515137, -0.614453136920929, 1.0118414163589478, -1.8313721418380737, 1.2100590467453003, 0.1559481918811798, -1.17607581615448, 1.493865728378296, 0.9122235178947449, 0.3185412287712097, -0.6421996355056763, -1.0491482019424438, 0.48142027854919434, 0.4699002504348755, -0.34866487979888916, 0.11534297466278076, 0.11756029725074768, -1.7860780954360962, -0.70631343126297, -2.5728213787078857, -0.650888979434967, 0.2932121157646179, -0.14379170536994934, -0.44419991970062256, -0.7052303552627563, -1.8442965745925903, -0.0026186006143689156, 1.4055540561676025, -0.783686637878418, -0.3144858479499817, 0.08754143863916397, -0.7300153374671936, -0.6659625172615051, -1.0741658210754395, 2.0037002563476562, -0.3049165606498718, 0.498706579208374, -0.20931701362133026, 0.48506003618240356, -0.16242794692516327, -1.8664740324020386, 0.9918018579483032, 0.22409917414188385, 0.6755396723747253, -1.6102701425552368, -0.2538532316684723, -1.2895574569702148, 1.2103502750396729, 0.1832578480243683, -0.07223282009363174, 0.6999274492263794, -1.9528361558914185, -0.3679683208465576, -2.1941945552825928, -0.7456581592559814, -0.03269338607788086, 0.4757837653160095, -1.1806678771972656, -0.6437615752220154, -1.0888876914978027, 0.9076600670814514, -1.2906217575073242, 1.977513313293457, -1.4248894453048706, -0.2766878306865692, -1.7468379735946655, 1.6745046377182007, 2.6201534271240234, 2.1952900886535645, 0.7068411707878113, -1.2944036722183228, -0.7106537818908691, 0.5128450989723206, -0.8481013774871826, 0.2609909772872925, 0.053397923707962036, 0.8763284087181091, 0.7384228110313416, -0.882125735282898, 1.410297155380249, 1.5490020513534546, -0.6445955634117126, -0.21955882012844086, -1.4661306142807007, -1.8852373361587524, -1.2866568565368652, -0.04524556174874306, -1.3056596517562866, 1.0045920610427856, 1.210650086402893, 0.10335323959589005, 0.31023430824279785, 0.16280490159988403, -1.6725181341171265, -0.7509099245071411, -0.9361861348152161, 0.3391355872154236, 0.6403661370277405, 0.19904643297195435, -0.33354926109313965, 0.39431995153427124, 1.3562777042388916, 1.186724066734314, -1.8702421188354492, -0.36405348777770996, 1.4269919395446777, 0.2158215343952179, 0.8441179394721985, 1.0433578491210938, 0.9146662950515747, -0.160169318318367, 0.19985099136829376, 1.2929502725601196, 0.44746220111846924, -0.6024254560470581, 1.1034871339797974, 0.9614633917808533, 2.1900527477264404, 0.8144389390945435, -0.30540725588798523, -0.2612689137458801, 1.1638672351837158, 0.6551349759101868, 0.9588111042976379, -0.9625936150550842, 0.3570133447647095, -0.10899044573307037, -1.491101622581482, -0.49799954891204834, -0.25119873881340027, 0.47735685110092163, -2.4178097248077393, -0.29290536046028137, -0.734415590763092, 0.7990200519561768, 0.7446930408477783, -0.8669844269752502, -0.2686074376106262, 0.05831689387559891, 0.06488418579101562, 0.11976244300603867, 0.9043865203857422, 0.38227856159210205, 0.1621537208557129, 1.540562629699707, -1.5380390882492065, -2.1347787380218506, -0.5115482807159424, 0.7183417677879333, 0.46316561102867126, -0.38525086641311646, -0.5542179346084595, 0.17039614915847778, -1.0751843452453613, -1.1753896474838257, -0.6142845749855042, 0.22826163470745087, 1.0615228414535522, 0.07475221902132034, -0.4806755483150482, -0.2791561782360077, -0.027424337342381477, 1.818177580833435, -0.9120476245880127, -2.034489154815674, 0.045767247676849365, -1.5125617980957031, -1.210903525352478, -1.7213867902755737, -0.20271112024784088, 0.02668493613600731, -0.2533538043498993, -1.3643224239349365, 2.2997257709503174, -0.49261942505836487, 0.2580893635749817, 0.5446261763572693, 0.88469398021698, -2.41251277923584, -0.46017134189605713, -1.083579182624817, 0.6922772526741028, -1.4491549730300903, 0.7317841053009033, 1.7365208864212036, -0.7818413972854614, -0.9256370067596436, -0.5879473090171814, -1.6136131286621094, 1.673087477684021, 0.7035831212997437, -0.26610881090164185, 0.0048683625645935535, 0.49516868591308594, -1.1055024862289429, 1.1540608406066895, -1.726123332977295, -0.4372202455997467, 0.627779483795166, -2.60246205329895, -0.7581477761268616, 1.3753207921981812, -2.0315916538238525, 0.8076616525650024, -0.38774800300598145, 0.6021091341972351, 0.3345000445842743, -0.3839362561702728, 0.8391343355178833, 0.7628829479217529, 0.8796834349632263, -0.4085443317890167, -0.635572075843811, 0.4867708086967468, -1.2172088623046875, -0.49764251708984375, 0.3131695091724396, -1.11045241355896, -1.4929286241531372, -1.179208517074585, -0.1811879426240921, -0.21693094074726105, 0.7584670186042786, 0.5051633715629578, -0.5553536415100098, 1.1385024785995483, 2.682969808578491, -0.6801338791847229, -0.4328521192073822, 0.6866705417633057, 0.9431864023208618, 1.2955076694488525, 0.31288275122642517, 0.1931968778371811, -1.7271677255630493, 0.5752845406532288, -0.2717442810535431, 0.15171675384044647, -0.8836275935173035, 0.2952757477760315, 0.7248769402503967, -0.36944839358329773, -0.3911087214946747, 0.1614830046892166, 0.6892272233963013, 1.6814919710159302, 0.08774887025356293, 0.835107147693634, -1.31936514377594, -0.12607380747795105, -0.20755639672279358, 1.7565077543258667, -0.7824769616127014, 0.7783611416816711, -0.983086347579956, -0.16039034724235535, -0.6789534687995911, 0.565912127494812, 0.07930789887905121, 0.14901693165302277, -0.5821247100830078, -0.867580771446228, -1.013770341873169, -1.4770209789276123, -0.9049655199050903, 0.22222396731376648, -0.7107160687446594, -0.5064471960067749, -0.5919530391693115, -1.143169641494751, 0.4321020841598511, 0.6544924378395081, 0.25982826948165894, -0.17579352855682373, 0.7321102619171143, -0.16568270325660706, 0.0898272842168808, -0.6935428380966187, 0.38521334528923035, -1.9252713918685913, -0.25069180130958557, -1.106871485710144, -0.24018892645835876, -0.15367580950260162, -0.11569369584321976, 0.4483188986778259, 0.4036218523979187, 0.03667612373828888, -0.7710909843444824, -1.389965295791626, -0.5018094778060913, 1.1378285884857178, 0.15523026883602142, -1.020869255065918, -0.2256268858909607, -0.8083890676498413, -0.01020634826272726, 0.7347500920295715, 0.8949581980705261, 0.3611548840999603, 0.654262125492096, 2.9731204509735107, -0.5275293588638306, -0.526314377784729, 0.5991294384002686, 0.579318642616272, -0.715953528881073, -1.0269417762756348, 1.5357301235198975, -0.5175204277038574, 0.29581719636917114, -0.6900025010108948, 0.5826054215431213, -0.035661838948726654, 1.1805790662765503, -0.18655456602573395, -1.0773979425430298, 0.45185282826423645, -0.8100791573524475, 0.14802773296833038, 1.5954396724700928, -1.525793194770813, 0.41261714696884155, -0.9038545489311218, 0.9110817313194275, -1.1296226978302002, 0.42565640807151794, -0.42129263281822205, -1.5080065727233887, 0.13977065682411194, 1.3707551956176758, -0.6701675653457642, -0.4193437993526459, -0.6705706119537354, 0.42757663130760193, -0.6602861285209656, -1.2635624408721924, 0.909692645072937, -0.05119461938738823, -0.9189491868019104, -1.9513660669326782, -0.4314071238040924, -0.0829230546951294, -0.2971014976501465, -0.19394314289093018, -0.06586875021457672, -0.12486469000577927, 1.3133147954940796, 1.4388892650604248, 0.38745400309562683, -0.158491849899292, -1.3363672494888306, 1.3928338289260864, 0.8860833048820496, 0.35187238454818726, 1.1307865381240845, -0.7166801691055298, -2.3824479579925537, -0.46947115659713745, -0.05516938120126724, -0.9449834823608398, -1.1732091903686523, -1.005265235900879, -1.3295269012451172, -0.8112343549728394, 0.491018146276474, -0.7496758699417114, 0.8730417490005493, -0.6929380893707275, 0.04200247675180435, 0.616601824760437, 0.25667741894721985, 0.22616912424564362, -0.7615681886672974, 0.42927226424217224, -2.0598933696746826, 1.3082722425460815, 0.02545136585831642, -0.5827632546424866, 0.46957001090049744, 0.3491116166114807, 0.22992441058158875, 1.263440489768982, -0.3097250163555145, 0.5093475580215454, -0.5124402642250061, 0.8224412202835083, 1.1051713228225708, 0.32075557112693787, -1.5096535682678223, -0.4415576756000519, -0.5606487393379211, 0.714267373085022, -0.26662421226501465, -0.4700201451778412, 0.0954015851020813, -1.1531555652618408, -0.7929217219352722, -0.7074044346809387, 1.1924132108688354, 1.6540007591247559, -0.46158847212791443, -2.537369728088379, 0.8022938966751099, 2.6979477405548096, 0.16825421154499054, -1.0072557926177979, 0.7782126665115356, 2.407788038253784, -0.44152408838272095, 0.794363260269165, 0.08157745003700256, 0.652448832988739, 0.3231199085712433, -0.7819058895111084, -2.2926113605499268, -0.7668956518173218, -0.8690305948257446, 0.5234013795852661, -0.04532702639698982, -0.14229901134967804, -2.3474619388580322, 1.1440019607543945, 0.7781294584274292, 0.35842955112457275, -1.661625862121582, -0.8164030313491821, -0.33426323533058167, -1.1798559427261353, 0.08834236860275269, -0.8004852533340454, -0.9328039288520813, -0.6275789141654968, -0.4805404245853424, -1.4143564701080322, 0.16788659989833832, 0.23871491849422455, -1.507155179977417, -0.08709368109703064, 0.6626346111297607, 0.6800066828727722, 0.8282456398010254, 1.8713935613632202, -1.5233572721481323, -0.6750756502151489, -0.007379462476819754, 2.0119147300720215, -1.3439760208129883, -1.393462061882019, 0.1468522995710373, -0.9740272760391235, 1.0051801204681396, -1.394439458847046, 0.814808189868927, 1.3944480419158936, 0.029931502416729927, -1.1434632539749146, -0.9518120884895325, -1.4877076148986816, 0.40260863304138184, -0.9917250871658325, -0.8691779375076294, -0.18770460784435272, -0.5306429862976074, -0.5004155039787292, -0.5881452560424805, 1.0770741701126099, 0.599301815032959, 1.4557877779006958, -1.6328661441802979, -0.22302831709384918, 0.7344024777412415, -2.459054470062256, 0.23995281755924225, 1.3310679197311401, -0.8690817952156067, -1.1139261722564697, -0.19731955230236053, -1.329372763633728, 0.6925650238990784, 1.6783305406570435, -0.0632738545536995, 0.45292308926582336, 0.13058003783226013, 0.018285881727933884, -0.2722990810871124, 1.8539342880249023, -0.6560680270195007, 0.6848991513252258, -0.42525801062583923, 0.6006526947021484, -1.6019755601882935, -0.41493672132492065, -0.8301069736480713, -0.03383464366197586, -0.633563756942749, -0.04205378144979477, -0.2977919280529022, 0.1153082624077797, 0.6680923700332642, -0.6287606358528137, -0.5153587460517883, 0.6655397415161133, 0.3099409341812134, -0.6114574670791626, 1.0011191368103027, -0.5912505388259888, 0.2676181197166443, 0.27958250045776367, -0.8981634378433228, -0.5595712065696716, 2.1813814640045166, 0.3115862309932709, -1.5072059631347656, 0.29681092500686646, 0.7642282843589783, -0.18020372092723846, -1.4723575115203857, 1.4288649559020996, 0.2472907453775406, 0.747380793094635, 0.670278012752533, 2.0243866443634033, 0.9563738703727722, -0.371898889541626, 0.26687538623809814, 0.7869671583175659, -1.4713767766952515, -0.34026452898979187, 0.18463271856307983, -0.16412010788917542, 0.08834867179393768, -0.11581477522850037, 0.9551172256469727, 0.2647736072540283, -0.649961531162262, -0.5809997916221619, -1.1468534469604492, 1.1894071102142334, -0.7232083082199097, 0.9254417419433594, 0.7544734477996826, 0.700964093208313, 1.4698092937469482, -0.4884399175643921, 0.30480119585990906, 0.5942946672439575, 1.1439025402069092, 1.1658897399902344, -1.1206380128860474, -0.9099759459495544, -0.5386897325515747, 0.34459587931632996, 0.767836332321167, -1.7232192754745483, 0.6835975050926208, -0.0013510261196643114, -0.0760856494307518, -0.032877881079912186, -0.4311061203479767, -1.1532984972000122, 0.9033189415931702, -0.0787748247385025, 0.5308095216751099, 0.9859645366668701, -0.23969537019729614, 0.5521576404571533, 1.1558558940887451, -0.3685036599636078, -0.13488981127738953, -1.8169231414794922, 0.08777172118425369, 2.0383949279785156, -2.132319927215576, 0.4053206145763397, -0.6095038056373596, -1.8432114124298096, -0.4776814877986908, -1.7779810428619385, 1.0318801403045654, -0.49203163385391235, 0.6397702693939209, 0.2189183533191681, 1.7105827331542969, 0.11813739687204361, -0.5915146470069885, 0.24756614863872528, -0.1879207193851471, 1.4942172765731812, 0.7764521241188049, 0.20716658234596252, 1.0375537872314453, -0.7432183623313904, 0.5962555408477783, -0.581839919090271, 1.118969202041626, 0.2511967122554779, -0.6512329578399658, -1.2020373344421387, 0.8068157434463501, -1.2608102560043335, 0.25184011459350586, 1.293735146522522, -1.8002556562423706, -0.937947154045105, -0.65419602394104, 0.07531167566776276, 0.8421508073806763, 0.5491850972175598, -0.17780274152755737, -0.5808789134025574, -1.081386923789978, -2.0552265644073486, 0.23410096764564514, 0.8012523651123047, -0.4871698021888733, -0.4503459334373474, -0.826261043548584, -0.4913701117038727, 0.5697086453437805, 0.7324396371841431, -0.7522843480110168, 0.9586958289146423, -0.38285720348358154, -0.7279345989227295, -0.06264331936836243, 0.03794795274734497, -1.4257686138153076, 0.11001800000667572, 1.4165726900100708, 1.075786828994751, 0.18861067295074463, -1.6468513011932373, -0.3449358344078064, -0.5552637577056885, -0.5943213105201721, 0.6350532174110413, 0.8162834048271179, 0.9170793890953064, -0.6423636674880981, -0.5393632054328918, -0.4593476951122284, 0.10792767256498337, 0.07041416317224503, 1.2726314067840576, -0.23718984425067902, -1.0739339590072632, -0.1460539996623993, 0.23054732382297516], [-0.13106052577495575, -1.808879017829895, -2.0938854217529297, 1.4488379955291748, 1.7497597932815552, -0.038875896483659744, -0.6609870791435242, 1.0436370372772217, -0.1861250251531601, 0.12273753434419632, -0.6627830862998962, 0.9160310626029968, 1.3659173250198364, -0.05382488667964935, 0.5230099558830261, -1.3207981586456299, -0.7530316114425659, 0.34033238887786865, 1.0285857915878296, 0.17612795531749725, 1.659072995185852, 0.13084964454174042, -1.4811439514160156, 1.1485798358917236, 0.34006544947624207, -0.781676709651947, -0.37529605627059937, -0.36497655510902405, 1.1488702297210693, 0.06308453530073166, -0.8050936460494995, 0.7361524701118469, 1.343683123588562, 0.35652458667755127, -0.3525242209434509, 0.36129215359687805, 0.18197371065616608, -0.5450996160507202, 0.1762191504240036, 0.6003953218460083, 1.0531103610992432, -0.437013179063797, 0.2207571417093277, 1.316572666168213, 0.37260743975639343, -0.6067744493484497, -0.6133158802986145, -0.3283484876155853, 1.437754511833191, 0.17967747151851654, -0.21354083716869354, -0.3230918049812317, -1.2829755544662476, -0.5695395469665527, -0.14447875320911407, -0.13652119040489197, 0.37960827350616455, 1.0906239748001099, 0.9324179291725159, -1.0545686483383179, -0.49219539761543274, 1.1837817430496216, -1.4457683563232422, -0.21328464150428772, -0.7528530359268188, 2.4091954231262207, -0.731354832649231, -0.8969427347183228, -0.6375834941864014, 1.3290644884109497, -0.8425911068916321, 0.6745010018348694, -1.6930221319198608, -1.215761661529541, -0.5699840188026428, 1.4709210395812988, 0.5660372376441956, 0.004957470577210188, 0.5220339298248291, 0.8094691038131714, -0.5930862426757812, 0.3935492932796478, 0.57292640209198, -0.3950899541378021, -1.121315360069275, 0.8328859210014343, -0.7606382369995117, -1.016432762145996, -0.9914407134056091, -1.306475043296814, -1.912107229232788, -0.13892073929309845, 0.5545249581336975, -0.21300433576107025, 1.009224534034729, -0.2889882028102875, 0.7441108226776123, -0.8759968280792236, -0.032590288668870926, 0.14255547523498535, -0.5411710739135742, -0.4643819332122803, 0.07937530428171158, 1.2946317195892334, 1.4544243812561035, 0.5644127726554871, 1.019209861755371, 0.3566196858882904, -1.7683380842208862, -0.7901359796524048, 0.4160018563270569, 0.1174657866358757, -0.6115875840187073, -0.12568068504333496, -0.14534138143062592, -1.482664942741394, -1.3531149625778198, 1.441223382949829, -1.0458893775939941, -2.7387914657592773, 0.9620115160942078, -0.6854206919670105, -1.4497501850128174, 0.6116641163825989, -0.8166005611419678, -0.7147146463394165, 2.161182165145874, -0.4010694622993469, -0.7532950639724731, -1.0143388509750366, -0.744464099407196, 0.2392551749944687, 0.4921560287475586, 0.5077250599861145, -0.2769271731376648, 0.24110807478427887, 0.05838388204574585, -0.3449316620826721, -0.6827762126922607, -0.266838401556015, -1.3625593185424805, 0.7716732025146484, 2.4196560382843018, 1.7279068231582642, 0.24648621678352356, 0.30622169375419617, 1.2404570579528809, 0.9122575521469116, -1.6606110334396362, 2.0415356159210205, -0.49241891503334045, 0.11989829689264297, 0.3262070119380951, -1.0774040222167969, 0.7474439144134521, 0.8225653171539307, -1.4835562705993652, 0.8492356538772583, -0.2779477834701538, 1.4855835437774658, -0.401782751083374, 0.6851376891136169, -0.12488161772489548, 0.6241342425346375, 2.3910582065582275, 0.6461514830589294, 0.12934596836566925, 0.5814123749732971, -1.235256314277649, -0.8564125895500183, -0.08067027479410172, -0.7492024302482605, 0.310716450214386, -1.2393877506256104, -0.7327681183815002, -0.7595661282539368, 1.0408015251159668, 0.8480197191238403, 1.2034060955047607, 0.75941401720047, 0.8729233741760254, -0.12700743973255157, 0.6300671100616455, -0.3491474390029907, 0.6671649217605591, -0.6132753491401672, 0.20613528788089752, -2.4613196849823, -0.665919840335846, 0.19457226991653442, -1.5603545904159546, 0.23528411984443665, -0.9244081974029541, -1.3239251375198364, -0.055906813591718674, 1.8553597927093506, -2.1637072563171387, 2.4665729999542236, 1.6595075130462646, 1.97793710231781, -0.7088028192520142, -0.03206530958414078, -0.257170706987381, 0.957066535949707, 2.0103490352630615, -0.2220347821712494, -2.018880844116211, -0.21383684873580933, 1.2468384504318237, -1.0518884658813477, 1.2524218559265137, -0.9506540894508362, -0.656561553478241, 2.039170742034912, 0.34495308995246887, 0.9557362198829651, 1.5138405561447144, -0.8632947206497192, -1.269548773765564, 1.0179063081741333, 0.8074008822441101, 0.23048727214336395, 1.1454002857208252, -0.6701674461364746, 0.30100372433662415, -0.2678791880607605, 0.5476701855659485, 0.7503052949905396, -0.1341101974248886, -0.3226346969604492, -1.74430251121521, 0.4498015344142914, 0.10780797153711319, -0.3345126509666443, 0.6194790601730347, 1.0853222608566284, 2.3688321113586426, 0.35030627250671387, -0.9792682528495789, -1.0011999607086182, -0.8880288004875183, -3.033364772796631, -0.8841741681098938, -0.31968966126441956, -0.8284607529640198, 0.4923853576183319, -0.536244809627533, -0.6763612031936646, -0.33717039227485657, 0.029436275362968445, 0.026061199605464935, 0.2962382137775421, -1.0074162483215332, -0.5917360782623291, 1.3952645063400269, 1.1561087369918823, -1.9928607940673828, -0.1689966917037964, -1.3790870904922485, -0.11532136052846909, -0.5158728361129761, -1.673177719116211, -0.06043936684727669, 1.6299166679382324, -0.5986287593841553, -0.12781532108783722, 1.9404925107955933, -0.35973870754241943, 0.49254488945007324, 0.09704513847827911, 1.8772648572921753, -0.2858579158782959, 0.09065688401460648, -0.0695355162024498, -2.012953042984009, -1.4838734865188599, 0.570983350276947, -0.21269331872463226, -0.7998704314231873, -0.5888376832008362, 0.7081761956214905, 1.6751002073287964, -0.2181491106748581, -0.7117338180541992, 0.45654478669166565, 0.4385323226451874, 0.03665492311120033, -0.45082056522369385, -0.6216684579849243, -0.27470943331718445, -0.3619271516799927, 2.416123390197754, -0.06601980328559875, 0.7111949920654297, 2.1134307384490967, 2.506502628326416, -0.9097317457199097, 0.5482292175292969, -0.9799320101737976, -0.004813441075384617, -0.580100417137146, 0.45258769392967224, 0.3920705020427704, 0.5821478962898254, 1.8207387924194336, -0.446971595287323, -1.2010233402252197, -0.31743699312210083, 0.8510706424713135, -0.6278946399688721, -0.24734146893024445, -1.3419315814971924, 1.1986817121505737, 0.7372816205024719, 1.4961462020874023, -0.7308542728424072, 0.187180757522583, -1.3336352109909058, -0.28856316208839417, 0.26530659198760986, 0.8381154537200928, 1.489789605140686, -0.5062291026115417, -0.17482802271842957, -1.5987838506698608, 1.5178685188293457, 2.4004132747650146, -1.3411396741867065, -2.402026891708374, -0.6092609167098999, 1.7956688404083252, -1.7947916984558105, -0.6153992414474487, 0.8475127220153809, -0.46462059020996094, -0.5527145266532898, -1.3153798580169678, -0.49318599700927734, -2.8137423992156982, -1.4120217561721802, 0.5743797421455383, -1.0110280513763428, 1.5882987976074219, -0.9580060839653015, -0.41866588592529297, 0.30043458938598633, -2.307969093322754, 0.8916354775428772, 0.30035948753356934, -0.5775653719902039, 0.986818253993988, -1.3726516962051392, -1.2518507242202759, 0.17789946496486664, 1.2734525203704834, -0.29218265414237976, -0.3248452842235565, -1.0490915775299072, 1.6214313507080078, -0.9739804863929749, -1.0200846195220947, -1.1229161024093628, -1.0287431478500366, 1.1259132623672485, 0.6126368045806885, -0.08723432570695877, -0.3622548580169678, -0.612959623336792, 0.4982748031616211, -0.9535693526268005, -0.07830479741096497, 0.5728296041488647, 1.0417066812515259, -0.09921054542064667, -1.076908826828003, -0.37742072343826294, -0.65471351146698, -2.256899356842041, -1.8263182640075684, 1.8492674827575684, 0.4911515414714813, -1.506492257118225, -0.39377808570861816, 0.011715204454958439, 0.3644779324531555, 1.0198010206222534, -1.2358295917510986, 1.50639808177948, -0.6750238537788391, -0.16529953479766846, -1.2108384370803833, -1.3188581466674805, 0.34885403513908386, 1.369242548942566, -1.0594505071640015, -0.8883765935897827, 1.8730438947677612, -0.301531046628952, -0.6729100942611694, -2.4437623023986816, -1.312943935394287, 0.7113633155822754, -0.6649743914604187, 1.439390778541565, -0.08477427065372467, 0.5747250318527222, 0.6384344696998596, 0.17721842229366302, 0.8031429052352905, 0.1996704787015915, 1.244657278060913, -0.7198191285133362, 1.4756397008895874, 0.6247212290763855, 0.925148606300354, 1.428640604019165, -2.0944454669952393, -1.6829148530960083, -0.19022968411445618, -0.802121639251709, 0.4288492500782013, -0.6118940114974976, -1.0832908153533936, -1.3037700653076172, 1.7565836906433105, -0.062301717698574066, 0.16190503537654877, 0.5934984087944031, -1.73552668094635, 0.12235165387392044, -0.24515295028686523, -1.8941388130187988, 1.3835428953170776, 1.2338287830352783, -1.3555006980895996, 0.2577787935733795, -0.4076007902622223, -1.2890337705612183, 0.9527133703231812, -0.4664720296859741, 0.353525847196579, 0.8407586216926575, -1.85360848903656, -1.2532434463500977, 1.783739686012268, 0.0109059177339077, -1.116458773612976, 0.08618289232254028, -0.2285887598991394, 0.5381507277488708, -2.3214898109436035, 0.1517239660024643, 0.28165221214294434, -0.5212935209274292, -0.19810450077056885, -0.8464866280555725, -0.06214234605431557, 1.5089998245239258, 0.4245958924293518, 0.07249803096055984, 0.792840838432312, 0.4806273281574249, 1.2705575227737427, -1.5956881046295166, 1.6940425634384155, 0.5052157640457153, -0.3716259300708771, -1.0820941925048828, 0.5288985371589661, -0.49241358041763306, -1.0961765050888062, 0.08603975921869278, 0.15243534743785858, 0.6961300373077393, 2.086806535720825, 1.358280897140503, 0.6493339538574219, -0.37453600764274597, -0.2377806305885315, 0.18314066529273987, 0.11763208359479904, 1.1700341701507568, -2.0211122035980225, -2.588033676147461, -0.1746273934841156, 0.41934502124786377, 0.6331600546836853, 1.3678903579711914, -0.08692912012338638, -1.078861117362976, 0.33967462182044983, -1.3564635515213013, 1.7461673021316528, 1.924368143081665, 0.4754835367202759, -0.8720264434814453, 0.5708710551261902, -0.6325066685676575, -0.9108805060386658, 0.5900785326957703, 0.05521345138549805, 1.439504861831665, 0.9846665263175964, 1.8754305839538574, -1.3255057334899902, 0.6246910691261292, -0.7253760695457458, -0.22974149882793427, -0.2814466953277588, 2.7311594486236572, 1.3024978637695312, -0.3858327865600586, 1.1515514850616455, 0.3296280801296234, 0.5512611269950867, 0.959343433380127, 1.2347829341888428, 1.027662992477417, 0.6023288369178772, -0.3667770028114319, -0.09101015329360962, -0.6317988038063049, -0.9033182263374329, 2.6996724605560303, -0.8775887489318848, 1.0659958124160767, 0.36558645963668823, -0.31891316175460815, -1.0817365646362305, -1.7315365076065063, -0.5980250239372253, -0.15370279550552368, -0.19633738696575165, -0.2579556703567505, 0.0791928842663765, -0.047403763979673386, 0.6858648061752319, -0.1268135905265808, -2.4283785820007324, -0.013650042936205864, -1.3311290740966797, 0.6720463037490845, 0.25879374146461487, -0.9029190540313721, -0.9245327711105347, 0.3320322632789612, -0.6650397181510925, -1.674835205078125, -0.17264658212661743, -0.1782752126455307, -1.4582574367523193, 2.0388803482055664, -0.10558349639177322, -1.3919225931167603, -0.2866891324520111, -0.17892779409885406, -0.7749002575874329, 1.7199167013168335, 0.43320199847221375, -0.17973725497722626, 0.7551098465919495, -1.0668293237686157, -1.5881696939468384, -0.8677124381065369, -0.03844446688890457, -0.66150963306427, 1.5235569477081299, 0.7591913938522339, 1.1054458618164062, 0.2574101984500885, 0.11998262256383896, 1.9650853872299194, -0.24689431488513947, -1.1990265846252441, 0.016979170963168144, -0.4221002161502838, 0.4422941505908966, -0.18310362100601196, -0.31540995836257935, 0.34956812858581543, -0.3540480136871338, 2.0731613636016846, 1.3121769428253174, -0.14132125675678253, -0.01711636409163475, 1.0700929164886475, -0.40151482820510864, -1.5468828678131104, -1.3906351327896118, -0.5834238529205322, -0.21734113991260529, -1.5141878128051758, -1.2647593021392822, -0.476094514131546, 0.49564775824546814, 0.5427748560905457, 0.060562897473573685, -0.42594921588897705, -1.4142464399337769, -1.3500417470932007, -0.6709303259849548, -0.49015098810195923, 1.3240045309066772, -0.6790187954902649, 1.1451889276504517, 0.15335972607135773, 0.2963820695877075, -1.1055126190185547, 0.4418765902519226, -0.026941249147057533, 1.5263525247573853, -0.5189201235771179, 0.18261757493019104, 0.5192906856536865, 2.2878000736236572, -0.2059248983860016, 1.4026973247528076, -1.3395512104034424, 1.1584798097610474, 1.5325868129730225, -0.5631235837936401, 0.4005663990974426, -1.938143253326416, 0.8552559018135071, 1.0181236267089844, 1.5619287490844727, 0.764092206954956, 2.093078374862671, -2.511549949645996, 0.16592277586460114, -1.6732569932937622, -0.09425654262304306, 0.6773346066474915, 1.4835712909698486, -1.2289650440216064, 1.2125427722930908, 0.017111625522375107, -1.0851136445999146, 2.016160249710083, -2.079866647720337, 0.528412401676178, -2.1082043647766113, 0.8526840209960938, -0.9537038207054138, 0.23370924592018127, 2.1354663372039795, -0.28818053007125854, -1.4503207206726074, -0.6607409715652466, -1.1772326231002808, 0.09463061392307281, -0.8490931391716003, -0.05136525258421898, -0.09840381145477295, 0.8963678479194641, -0.6150646209716797, 0.2673921585083008, 0.49705490469932556, -0.7730176448822021, -0.6188576221466064, -1.1549235582351685, -2.681269645690918, -1.0148675441741943, -0.9917877912521362, -0.9201709628105164, 0.5097981095314026, -0.05155406519770622, -0.16986368596553802, 2.622359275817871, 0.21118833124637604, 0.8206445574760437, -0.9428359270095825, 1.2223159074783325, -0.6950573325157166, 0.6854553818702698, 0.4738265872001648, 1.9375584125518799, 0.2522139251232147, 0.18097367882728577, 0.09739314764738083, 0.8537794351577759, 0.3348690867424011, -0.43136516213417053, 1.9560191631317139, 0.05477740243077278, 1.7410252094268799, -1.3975690603256226, -0.5313844680786133, 0.39551842212677, 1.2608214616775513, -2.390202045440674, -0.9733461141586304, -1.5670026540756226, 0.3973311185836792, -0.7528475522994995, -0.6521236300468445, -0.5389844179153442, 1.185292363166809, 0.8812993168830872, 0.7843057513237, -0.8678072690963745, 0.5906404852867126, 0.5302391648292542, 0.9429073333740234, 0.6334992051124573, 0.5474304556846619, -1.3245130777359009, -0.12248991429805756, 1.078168272972107, 0.7043605446815491, 0.7143640518188477, -0.05000780150294304, -0.5765290856361389, 1.2609803676605225, -0.9671590328216553, -0.5895830392837524, -0.37998029589653015, 1.3288766145706177, -0.6548339128494263, 0.25801724195480347, -2.119211435317993, 0.21042391657829285, -0.5565779805183411, -1.184160590171814, 1.250215768814087, -0.13828352093696594, 0.42098885774612427, -0.7907026410102844, 0.24541836977005005, 0.6313925385475159, -0.6598489880561829, 1.8268040418624878, -1.3598798513412476, -0.9572213888168335, -1.0392036437988281, 1.4044777154922485, 0.6413853168487549, 0.46316686272621155, -1.3265151977539062, 2.264958143234253, -0.4158059060573578, -0.35111311078071594, 1.5896868705749512, 0.6282795071601868, 0.6774654388427734, -1.3701744079589844, -1.8132290840148926, 0.4415774345397949, 1.1829496622085571, -0.37255486845970154, 0.8925327062606812, -0.4350427985191345, 0.8337840437889099, -0.3758850395679474, 2.238770008087158, -1.0042970180511475, 0.07540429383516312, 0.30813440680503845, -0.49914851784706116, 0.35521385073661804, -0.4299711585044861, -0.3297044634819031, 1.1065515279769897, -0.6998267769813538, -0.4672335982322693, -0.8496696352958679, -0.0035799311008304358, -0.5949289798736572, 0.4282898008823395, 1.4768918752670288, -0.9310141205787659, 0.22373102605342865, 1.4308468103408813, 0.7445754408836365, -0.6867515444755554, 1.1174179315567017, 0.6818594336509705, -0.11295153945684433, 1.7060133218765259, 0.26793238520622253, 0.5394641757011414, -0.2635420858860016, 0.10092166066169739, -0.6019817590713501, 0.12032289057970047, 1.7189425230026245, 0.4463109076023102, -1.1039613485336304, -0.05441403388977051, 0.8969255685806274, 0.4114837050437927, -0.680906355381012, 0.6936559677124023, 0.6934728622436523, -0.7070108652114868, -0.3934609889984131, 0.8300220370292664, -0.47724980115890503, -0.08683884888887405, 0.9168877601623535, -0.0732438713312149, -0.3813795745372772, -0.6359172463417053, -0.2838118374347687, 0.9710109233856201, -0.09336449950933456, -0.7554435729980469, -0.2637166678905487, -0.14549003541469574, -0.2772422432899475, -1.388024926185608, 0.5253230333328247, 0.9711868166923523, -0.6436231136322021, -2.549485445022583, -0.5317500829696655, -0.9843344688415527, 1.5851441621780396, -1.837531566619873, 0.5232495069503784, -0.19746136665344238, 0.1582339107990265, -0.44950783252716064, -0.18204186856746674, 0.547275722026825, 0.9460591077804565, 0.132583349943161, -1.4998843669891357, 0.2617368996143341, -0.5137216448783875, 0.1404201239347458, 0.8978462219238281, -1.0531176328659058, -0.6998589634895325, 0.4374684989452362, -0.20291262865066528, -0.4173702895641327, 0.9231467843055725, 0.9354698061943054, 0.1640292853116989, 0.28357410430908203, -1.3385975360870361, 0.16782940924167633, -0.37711501121520996, 0.23571453988552094, -0.047405943274497986, 0.8622344732284546, -0.03490546718239784, -0.45905497670173645, -1.777382493019104, -0.7365648746490479, 0.6646602153778076, -0.3232250213623047, -0.019752953201532364, -0.29257556796073914, -1.309908390045166, -1.396708369255066, -1.1842883825302124, 1.5215872526168823, -1.4185625314712524, 0.4789150655269623, -1.261194109916687, -1.0652776956558228, 0.1364940106868744, 0.09306831657886505, -0.9738923907279968, 0.44791823625564575, -1.1312795877456665, 0.3243277370929718, -1.0825392007827759, 0.306938499212265, 1.801681637763977, -1.0968055725097656, -0.6584792137145996, -0.8016167283058167, -0.010165378451347351, -1.021248459815979, 1.6576988697052002, 0.6748186349868774, -1.0926227569580078, -0.8529926538467407, -1.054261565208435, 1.097886085510254, 0.6179085969924927, -0.36432889103889465, -1.3430768251419067, 0.8364017009735107, 0.4235299229621887, -0.47324109077453613, 1.8871610164642334, -1.2155147790908813, -0.014059496112167835, -0.7265192270278931, -0.03680083900690079, -0.7169342637062073, -0.5094781517982483, -0.7327842712402344, -1.1460716724395752, 0.37202703952789307, -1.261272668838501, 1.0192407369613647, -1.2582038640975952, 0.6384567022323608, 1.3537967205047607, -0.18473508954048157, 1.7598994970321655, -0.7564025521278381, -0.40379175543785095, 0.5071122050285339, -0.3664303719997406, -1.3669393062591553, 1.7506928443908691, -0.5318968892097473, 1.6655687093734741, -1.0029246807098389, 1.3783059120178223, -0.8211954832077026, 0.5190315842628479, 0.721488893032074, 0.280097097158432, -1.1323202848434448, 0.35021987557411194, 0.08322981745004654, -0.03857959061861038, -0.3797752857208252, 0.08616512268781662, -0.05352912098169327, -0.8110247254371643, -1.1441559791564941, 0.6804028153419495, -0.09542514383792877, 0.4653436243534088, 1.6259334087371826, -0.5988554954528809, 0.45559316873550415, 0.9161257147789001, 0.1062423586845398, 0.2158886194229126, 2.202633857727051, 0.6035668253898621, 1.1394281387329102, 0.45360153913497925, -0.6233402490615845, -0.11110451072454453, 1.6187556982040405, -0.9782525897026062, 0.05454880744218826, 0.8810582160949707, 0.7435581088066101, -0.5904102921485901, -2.001922369003296, 0.36573293805122375, -1.364383578300476, -0.34117481112480164, 1.2502822875976562, 1.1537753343582153, -2.028108835220337, -1.1546893119812012, -0.22262728214263916, -0.4378572106361389, -0.016116177663207054, -0.3627234399318695, -0.4578844904899597, 0.2638436555862427, -1.1003631353378296, 0.6698618531227112, -0.5280863642692566, 0.054894644767045975, 1.0064969062805176, 0.07997873425483704, -0.5194991230964661, 0.4388216733932495, 0.4327641427516937, -0.15304803848266602, 0.13610723614692688, -0.009548661299049854, -1.0432610511779785, -0.884178876876831, -0.5075405836105347, 1.6259580850601196, 0.0872233435511589, 1.94606614112854, 1.4580395221710205, 0.8129401206970215, -0.8846266269683838, 0.8513160347938538, 0.184127077460289, -0.08884429931640625, 0.2282647043466568, 1.631726861000061, 0.3752037584781647, 1.3394194841384888, -1.760109305381775, 0.13113006949424744, -1.3210424184799194, 1.7704312801361084, -1.0548315048217773, -0.32324478030204773, 0.708410918712616, -0.3092298209667206, 1.0273411273956299, 1.3690434694290161, 0.3386356830596924, -2.194390058517456, 0.06092150881886482, 1.320328950881958, -1.7810251712799072, -0.5297194719314575, 0.16830061376094818, -0.39365360140800476, -0.11423257738351822, 0.9849968552589417, -1.1410682201385498, -2.874633312225342, -0.48383697867393494, -0.38048917055130005, 1.2337961196899414, -0.7554543018341064, -0.2839087247848511, 2.995905876159668, 0.26169878244400024, -0.6470032334327698, -0.00853260513395071, -0.17879562079906464], [0.2591259181499481, -0.26802992820739746, 1.0160868167877197, 1.1216903924942017, 0.2403186559677124, 0.2421160191297531, -1.2798551321029663, 1.0243197679519653, 1.8659789562225342, -0.35248470306396484, -1.238993763923645, 0.11872659623622894, -1.495445728302002, 1.7885760068893433, 1.0479457378387451, -0.3691149950027466, -0.07981783896684647, -0.2102019488811493, 0.6272245645523071, -1.1563960313796997, 0.8792152404785156, -0.15981334447860718, -0.5992421507835388, 1.4318478107452393, 1.280221700668335, -1.0759695768356323, 1.4413962364196777, -1.28226637840271, -0.9496459364891052, 0.7876049280166626, 0.18670450150966644, 1.8555186986923218, 1.2965127229690552, -0.12472759187221527, -0.6642933487892151, -0.2495443969964981, -0.6007506251335144, 0.9918071031570435, -0.13405436277389526, -0.0753306895494461, -0.1989486664533615, -1.7847596406936646, 1.1672061681747437, -0.05106527358293533, 0.2515134811401367, -1.052722692489624, 1.7333450317382812, 0.37288641929626465, 0.8748924136161804, 0.7353326082229614, 0.947620153427124, -1.5179836750030518, 0.7845024466514587, 1.0325734615325928, 0.4753722846508026, 0.11018585413694382, -0.6913334727287292, 0.17880184948444366, -1.187409520149231, 1.029172420501709, 0.1847149282693863, 1.4015153646469116, -0.9573583602905273, -0.7260577082633972, -0.28083744645118713, 0.008590876124799252, -0.3894329369068146, 1.0727940797805786, 0.9553707838058472, -0.1309673935174942, 2.3418190479278564, 0.16767854988574982, 0.20919042825698853, -0.07725170999765396, 0.18126611411571503, 0.5324316620826721, -0.9478914737701416, -0.9659174084663391, -0.4336087703704834, 1.0504413843154907, -0.8533512353897095, -0.7341095209121704, 1.1601872444152832, 0.4793424606323242, -0.2789627015590668, -0.12141580879688263, 1.027521014213562, 1.5414092540740967, -0.15981070697307587, 0.1770571619272232, -0.9396929740905762, -0.8740041851997375, 0.998568058013916, -1.252113699913025, 0.5735645294189453, 1.4732953310012817, 0.5342850089073181, 0.6026424765586853, 0.38839730620384216, -0.8225358724594116, 0.22417718172073364, 1.190407633781433, 1.7881568670272827, -1.5750218629837036, -0.09492150694131851, 1.149242877960205, 1.1733222007751465, -0.8949548006057739, -0.4233384132385254, 0.22939728200435638, 0.009050353430211544, -0.9389910101890564, 1.6448051929473877, -1.005419135093689, 0.782319962978363, -0.10163447260856628, 0.14290796220302582, 1.4060965776443481, 0.05966398864984512, -0.6584044098854065, 0.05597888305783272, -1.229384183883667, -0.9269996881484985, 1.193972110748291, 0.4697093367576599, 0.08926211297512054, -0.05412755161523819, 0.4573562741279602, -0.3056590259075165, -1.115593433380127, -1.0249583721160889, 1.3446099758148193, -1.9041962623596191, 1.2162529230117798, 1.1045652627944946, 1.2252899408340454, -1.7566502094268799, 0.24182067811489105, 1.0377986431121826, -0.6805087327957153, -1.963739037513733, -0.48735958337783813, -0.46176832914352417, 0.25186455249786377, 1.3382678031921387, 0.12662221491336823, -0.4782959520816803, -1.3179179430007935, -1.09685480594635, -1.991901159286499, 0.8330119848251343, -1.3054773807525635, 2.3485193252563477, -0.8646830320358276, 0.4712303876876831, -0.890803337097168, 1.0454624891281128, -0.9250736832618713, -0.29149752855300903, -0.036075059324502945, -0.1527317613363266, 0.1479680836200714, 0.4540037512779236, -0.39062419533729553, -0.43271806836128235, -0.6779000163078308, 0.3526029586791992, 1.5418474674224854, -1.197460651397705, -0.35808995366096497, -0.1536230742931366, -0.3724595308303833, 0.31476572155952454, -0.6911521553993225, 1.2363885641098022, 1.3904967308044434, 0.2886515259742737, -1.0402494668960571, 1.7143535614013672, 1.6379328966140747, -1.7169026136398315, -0.0996493548154831, 1.210850477218628, -0.143111914396286, 0.6493207812309265, 0.8251767158508301, -1.6762586832046509, 1.4144302606582642, -0.2826250195503235, -0.8070565462112427, -0.842542290687561, 0.5249122381210327, -0.019946910440921783, -2.9723923206329346, -2.5539870262145996, 0.11725985258817673, 1.0326259136199951, -0.21333132684230804, -0.31393542885780334, -0.1930648535490036, 0.42810699343681335, 1.5158919095993042, -1.0317965745925903, -0.38501977920532227, -0.21009321510791779, 1.0516493320465088, 0.6651054620742798, 0.12295454740524292, -1.049243688583374, -0.6984583735466003, 0.6572421789169312, 0.6623862385749817, 1.1437486410140991, 0.256237655878067, -0.49082720279693604, -0.0022726678289473057, 1.4378135204315186, 0.9996232390403748, -0.5246405005455017, -0.38196760416030884, 0.36312687397003174, 0.15605251491069794, 1.5539886951446533, 0.3202396035194397, -0.030585210770368576, -1.0506176948547363, 0.1872270107269287, 0.3343663811683655, -0.9680208563804626, -0.2975073754787445, 0.4563128650188446, 0.047527194023132324, -1.2045557498931885, 0.6972100734710693, -0.08323844522237778, 1.856234073638916, 0.020418692380189896, -0.5446091890335083, -0.46303030848503113, -0.6772364974021912, -0.7837697267532349, 1.0251126289367676, 0.7521235346794128, -0.25256526470184326, 0.5448136329650879, 1.4334471225738525, 0.6530514359474182, -0.6102855801582336, -1.2405531406402588, -0.5039632320404053, -1.6730787754058838, 0.2773662209510803, 0.9470948576927185, -0.9399136304855347, -0.4272744655609131, 1.2658618688583374, -1.1512929201126099, 1.680209755897522, -0.07264383137226105, 0.7079228758811951, 0.6524596214294434, 0.27580133080482483, -1.1124610900878906, 0.5585111975669861, -1.180950403213501, 0.16800089180469513, -0.09872908145189285, 1.020917534828186, -0.19938305020332336, 0.8287001848220825, -0.8483781814575195, -0.35428062081336975, 1.1072853803634644, -1.4305338859558105, 0.10798117518424988, 0.9526771306991577, 0.27855899930000305, 0.14101919531822205, -1.0182868242263794, -1.5171173810958862, 0.8650087714195251, 1.0017552375793457, -0.08513078838586807, 1.2701539993286133, 1.3391101360321045, -0.13509109616279602, 1.870619297027588, 0.33972299098968506, -1.14337956905365, -1.3340859413146973, -1.013505220413208, -0.22109262645244598, -0.20880916714668274, -1.7276595830917358, -0.2944280505180359, 0.45899373292922974, -1.352366328239441, -1.5428290367126465, -1.1247082948684692, -1.3463513851165771, 0.22967596352100372, 0.9557642340660095, -1.6740448474884033, 1.7107707262039185, 0.2244938611984253, 0.69550621509552, 1.0055583715438843, -0.4065285623073578, 0.5388802289962769, -1.3496596813201904, 0.6322316527366638, 0.6487866044044495, -0.9031796455383301, -0.23610779643058777, -1.4721089601516724, 0.905190110206604, 0.3750559389591217, 0.7996001839637756, 0.01301331352442503, -0.6295381188392639, 0.28743648529052734, 0.13613922894001007, 0.20210565626621246, 0.03228123113512993, -1.0189456939697266, 0.26836714148521423, 1.292943000793457, -0.6171175241470337, 0.2928520739078522, -1.0044454336166382, 1.5135868787765503, -0.29389309883117676, -0.8467987775802612, 0.6308833956718445, 0.422215074300766, -1.167972445487976, -1.1081547737121582, 0.3664238154888153, 1.0760153532028198, 0.05834050476551056, 0.5925350189208984, -0.8204813003540039, 0.6000108122825623, -0.9123000502586365, 1.3939886093139648, 1.1953731775283813, -0.8716278076171875, 0.6551865339279175, 0.41210228204727173, 0.09711416065692902, -0.4176013171672821, 0.24074633419513702, -0.45813146233558655, -0.7260051965713501, 0.386790931224823, 0.9938795566558838, 0.6682818531990051, 0.2891808748245239, 1.4721440076828003, -0.9243475198745728, -0.6950120329856873, 1.3716838359832764, -0.9067646265029907, 0.15585686266422272, -0.20418515801429749, -0.9631355404853821, -0.27452850341796875, -0.10455010831356049, 0.9977672100067139, 0.8056686520576477, 1.760794997215271, 0.4131069481372833, 0.08424884080886841, -0.10165167599916458, -0.6683822274208069, -0.51863694190979, -0.20847325026988983, -1.2461875677108765, 1.5837520360946655, 0.07594034075737, 0.9982271194458008, 0.11238022893667221, 0.2216905653476715, 0.09563294053077698, 0.9537835121154785, -0.33734720945358276, -0.5326592326164246, 0.5283808708190918, 0.39830610156059265, 0.8144713044166565, 1.1284528970718384, -0.2735847532749176, -1.6659960746765137, 1.0084691047668457, 1.4167529344558716, 1.8745834827423096, -1.1025798320770264, -1.2880080938339233, -1.2077713012695312, -0.34368664026260376, 0.4050246477127075, -0.5820713639259338, -0.2671644687652588, 2.2059810161590576, 1.3499058485031128, 0.3044445812702179, 0.6816965937614441, -0.7154958248138428, -0.45741406083106995, -0.6587576270103455, -0.06020035222172737, 0.147922083735466, -1.2374016046524048, -0.21360653638839722, -0.43678557872772217, 1.19412100315094, 0.9519258141517639, 1.0890471935272217, 0.547224223613739, -0.06556788831949234, -1.6344454288482666, 0.21278749406337738, 0.6839765906333923, 0.569970965385437, -0.5278494954109192, -0.31492772698402405, -0.5105937719345093, -1.1893137693405151, -0.948387086391449, -0.3165521025657654, 0.2720485031604767, 2.311201810836792, -0.6298847794532776, -0.4060632586479187, -0.010016089305281639, 0.13870353996753693, 0.811545729637146, 0.30670681595802307, -1.2287269830703735, -2.5437028408050537, 1.4132194519042969, -0.43930819630622864, 1.104024887084961, -1.7240530252456665, -1.1615554094314575, -0.6757712960243225, -1.2587286233901978, -0.21251629292964935, 2.0001790523529053, -1.043601393699646, 0.698977530002594, 0.6526820659637451, 1.1481343507766724, -0.32831066846847534, -1.265181303024292, 0.7052805423736572, 1.4831442832946777, -0.8579976558685303, -0.2512832283973694, 0.33496496081352234, -0.4619775116443634, 0.2774280309677124, 0.7862605452537537, -1.0059797763824463, 0.5295366048812866, -0.995803713798523, -0.2873936891555786, 1.1169100999832153, -0.02275526151061058, -0.4135454297065735, 0.36442503333091736, -1.4942103624343872, -0.7926980257034302, 0.11855582892894745, -0.1823207587003708, 0.9201536178588867, 0.6067147850990295, -1.439210057258606, -0.8333502411842346, -0.25587552785873413, -0.5589631795883179, 1.4896013736724854, -0.3527975082397461, -0.6734524965286255, 0.26459449529647827, -2.4818692207336426, -0.4975586533546448, 1.767491340637207, -1.6253352165222168, 1.5878770351409912, -0.3332878649234772, 0.5339417457580566, 0.24916847050189972, 0.20913510024547577, 0.080686055123806, -0.05665592476725578, 1.0380842685699463, 0.22267675399780273, 0.7291949987411499, 0.523803174495697, -0.7918144464492798, -1.4124280214309692, -1.5566537380218506, 0.8610028624534607, -0.36114931106567383, 2.0097107887268066, 1.8193517923355103, -0.3629276156425476, 1.462699294090271, 1.801357388496399, -0.4683774411678314, 1.2956831455230713, -0.9157276153564453, -0.5985320806503296, -0.5822930335998535, -1.982850193977356, 1.046455979347229, -0.3450087606906891, -1.2289644479751587, -1.1687597036361694, -0.8287710547447205, 1.336617350578308, 0.8005499839782715, 2.0732414722442627, -0.9961706399917603, 0.9408422708511353, 0.19302397966384888, 0.45300307869911194, 0.7600499391555786, -1.6501801013946533, -1.70011568069458, -0.5588716268539429, 0.49572935700416565, -0.0036835195496678352, -0.7511788010597229, 1.8587303161621094, -0.34045279026031494, 0.9567459225654602, -0.45495182275772095, -0.9422377943992615, 0.028943344950675964, 0.037963852286338806, 0.6123889684677124, 0.780348002910614, -1.6430950164794922, 1.494626522064209, 1.3074195384979248, -0.49466177821159363, 1.6914374828338623, -0.8101850748062134, 0.5415235161781311, -0.37461838126182556, 0.5935986638069153, 1.4435606002807617, -0.04717056080698967, -0.1591242551803589, 0.8896996974945068, -1.131508469581604, -0.08344513922929764, 0.519137978553772, 0.3988892436027527, -2.3403351306915283, 1.3213878870010376, -0.2740769386291504, -0.6457955241203308, -0.5311920046806335, -2.06842041015625, 1.3123440742492676, 0.5849570631980896, -0.2722504138946533, -0.09727278351783752, -0.07133200764656067, 0.7031986117362976, -1.528109073638916, -0.3603816628456116, 1.438266396522522, -0.6409218311309814, 0.5732085108757019, 0.2555442154407501, 0.6529554724693298, 0.8374777436256409, -1.1567894220352173, 0.977093517780304, -0.5907863974571228, -0.5626186728477478, 0.6161661148071289, -0.6703842878341675, 1.648325800895691, -1.1529566049575806, -1.6143895387649536, 0.6753509640693665, 2.2899622917175293, -0.625037431716919, -0.3826707899570465, -0.4166811406612396, 0.9707753658294678, -0.10290200263261795, 0.9808960556983948, 0.44362664222717285, 0.7001832723617554, 2.7741739749908447, -0.9526672959327698, 0.15537181496620178, -0.14529424905776978, 1.2190755605697632, -0.018544992431998253, -0.1679476946592331, -1.7540205717086792, 0.18214544653892517, 0.7151804566383362, -0.9208706617355347, -0.34829556941986084, 1.3806008100509644, 0.40210244059562683, 0.045363906770944595, -2.432713031768799, 1.059262990951538, 0.19456203281879425, -2.526275873184204, -0.012443538755178452, -0.9040161967277527, 0.8264160752296448, 1.2194812297821045, 1.0246951580047607, -0.5180010199546814, 0.5803800225257874, -0.2995876669883728, 0.07442207634449005, -0.13108323514461517, -1.3155320882797241, -3.4113645553588867, 0.3811902403831482, -0.33832547068595886, 0.9911361932754517, 0.4868559241294861, 0.8401615023612976, -0.24507737159729004, -0.13077880442142487, 1.6623203754425049, -0.5702005624771118, -1.992508888244629, -0.3872303068637848, 0.07873000204563141, 0.2043401449918747, -0.3436947166919708, -1.4132202863693237, -0.9490681290626526, 0.8884331583976746, -0.28571945428848267, 1.0834630727767944, 0.03263404965400696, 0.2827391028404236, 1.5373657941818237, 0.04209066927433014, -0.5206025838851929, -2.3676040172576904, 1.943734884262085, 0.10867089778184891, 0.7586131691932678, -0.2734048068523407, -1.086850881576538, -0.5201236009597778, -0.4418640434741974, -1.959029197692871, 0.27849632501602173, -0.13800473511219025, 1.0423763990402222, 0.20902207493782043, -0.5307218432426453, -0.3441762626171112, 1.0316760540008545, 0.06558012217283249, -0.17052580416202545, 2.2415237426757812, 1.0438122749328613, -0.37961894273757935, -0.0608706921339035, -0.5068091154098511, 1.1565769910812378, -0.4576590955257416, 0.6156125068664551, -0.455167680978775, -0.27205488085746765, 1.6503206491470337, 1.174025535583496, -0.0806761160492897, 0.33801236748695374, -1.0621329545974731, 0.5673292279243469, 2.4651503562927246, -0.04069170728325844, 0.587762713432312, -0.9131000638008118, -0.6415983438491821, -0.7515742778778076, -1.3589262962341309, 0.6158743500709534, 1.4246684312820435, 0.9629073143005371, 0.34786558151245117, 0.025326890870928764, 1.1104381084442139, -0.019764328375458717, 2.2123069763183594, 2.052480697631836, -1.5715388059616089, 0.983116626739502, -2.0979089736938477, 1.5378856658935547, -0.046411581337451935, -0.6101896166801453, 0.785896897315979, 0.45301854610443115, -1.0832998752593994, 0.47546932101249695, 2.25404691696167, 1.0932588577270508, -0.5037673711776733, 0.04718491807579994, -1.2047839164733887, -0.1674593985080719, -0.6487149596214294, 1.1122816801071167, 0.018744373694062233, -0.19590677320957184, 0.8910555243492126, -0.5822629332542419, 1.270118236541748, -0.21151359379291534, 0.7427029013633728, 0.5353590846061707, 0.6811496019363403, -0.16050802171230316, -1.2621417045593262, -1.6527067422866821, 0.6338938474655151, 0.43572431802749634, 0.16075211763381958, -1.7517799139022827, 0.8292118906974792, 1.188577651977539, 2.338545799255371, 1.163719892501831, 1.7860088348388672, 0.11847738176584244, 0.7794094681739807, -0.6374245882034302, 1.1615155935287476, 0.8452675938606262, -0.30100855231285095, -1.0539315938949585, 0.9241086840629578, -0.26616206765174866, -1.2962685823440552, -0.12701521813869476, 0.5738398432731628, 0.44553595781326294, 0.6518828272819519, -0.3221684396266937, -1.123054027557373, -1.1890008449554443, 0.13360832631587982, 1.96694016456604, -0.0014950408367440104, 0.3253900706768036, 0.09037373214960098, 0.05029076337814331, -1.246345043182373, -0.8065493702888489, 0.3903352618217468, -0.7237792611122131, -1.0240052938461304, -2.3549726009368896, -0.6503530144691467, -1.5762490034103394, -0.5076115131378174, -1.5314531326293945, 1.4245579242706299, -0.6166847348213196, -0.445854514837265, -0.8690741658210754, -0.3242361545562744, -0.4061095118522644, 1.632773756980896, -1.168104887008667, 0.9697859883308411, -0.30671846866607666, -0.462674081325531, 0.6624609231948853, -0.40842288732528687, -0.3915005028247833, 0.5326865911483765, -2.1162545680999756, -0.23406922817230225, 2.72651743888855, 0.3813849091529846, -0.3920959234237671, 0.9725589156150818, 1.0326313972473145, 1.8216955661773682, 0.9303016066551208, -0.0944906622171402, 0.23862725496292114, 0.6553936004638672, 0.0068217553198337555, -0.4361644387245178, -0.24371075630187988, 1.2502108812332153, 0.8598679900169373, -0.1460922360420227, 0.29396361112594604, 0.413350373506546, 0.6229326725006104, -0.2503790557384491, -0.4215804934501648, -0.6051079630851746, -0.019681256264448166, -1.3262977600097656, 2.054630994796753, 2.475569009780884, -1.375810146331787, -0.08587227761745453, 0.4118799567222595, -1.133242130279541, 1.4638168811798096, 0.8915777206420898, -0.7132524251937866, -1.2727046012878418, -0.11844126135110855, -0.05738712102174759, -0.8686826825141907, 0.7887016534805298, -1.3364843130111694, 0.2784057855606079, 0.7730264663696289, -0.19655412435531616, -0.5281285047531128, 1.3243147134780884, 0.6749520897865295, 1.3220187425613403, -1.0488930940628052, 0.7241488099098206, 0.333647757768631, -0.6266413331031799, -0.19508014619350433, -1.3473209142684937, 1.1491552591323853, 0.9210100173950195, 0.16280288994312286, -0.6408696174621582, -0.7810350060462952, 1.135318398475647, 0.21902111172676086, 0.04847220331430435, -1.3386105298995972, -0.7161526679992676, -0.08007831126451492, -0.6763702034950256, -1.1517601013183594, 0.015281430445611477, -0.3194470703601837, -0.622123658657074, 2.3855831623077393, -0.9349331259727478, 1.0933481454849243, 0.9212508201599121, 2.0746021270751953, -0.8951879143714905, -0.5878145098686218, -1.8231267929077148, 0.3175547122955322, -0.8177253007888794, 1.72853422164917, -0.426990270614624, 0.8979916572570801, 0.21200354397296906, -0.032856233417987823, 2.3055832386016846, 1.7920266389846802, 0.7584800124168396, -0.47321632504463196, 0.4592387080192566, 0.639157772064209, -1.3608218431472778, 0.024726789444684982, 0.18825283646583557, 0.3055536448955536, 0.5735899209976196, -0.4952857792377472, -0.8429421782493591, 0.9839181900024414, 0.9133939743041992, -0.5548391342163086, 2.0378453731536865, 0.21492719650268555, -0.9851693511009216, 0.14709952473640442, -1.5474179983139038, -0.015455484390258789, 0.245785653591156, -0.5343245267868042, -1.1868116855621338, 1.3084304332733154, -0.45023849606513977, 0.2597605884075165, -0.5856239795684814, 0.09782716631889343, 0.03720061853528023, 0.07257577776908875, 0.14656832814216614, 1.5625907182693481, 1.2096832990646362, 0.7521200776100159, -0.27897608280181885, -0.4591035544872284, -2.4381473064422607, 1.368125319480896, -0.14190633594989777, -0.5887492299079895, 1.967979073524475, -0.7853642702102661, -1.2002345323562622, -1.185315728187561, 0.5182487368583679, 0.45884057879447937, 1.0704989433288574, -0.8738382458686829, 2.386458158493042, 0.8069023489952087, -0.17494261264801025, 1.2390966415405273, -0.7657752633094788, -0.33545851707458496, -0.4447464346885681, -1.3469127416610718, 0.14332227408885956, 1.4611366987228394, 0.04266202449798584, -0.4992203712463379, 0.754538893699646, 0.678572416305542, -2.169696807861328, 0.44178664684295654, -0.44528523087501526, -1.6972651481628418, -1.953029751777649, 0.6512653231620789, -1.166013240814209, -1.7649438381195068, -0.7784968614578247, -0.018878575414419174, -0.9332787990570068, -0.2681472599506378, 1.0100853443145752, 0.4908589720726013, -0.30388209223747253, -1.0559051036834717, 0.481182336807251, 3.396979331970215, -0.22986096143722534, 0.8554006218910217, 1.4592515230178833, 0.6967501640319824, 0.812480092048645, 1.9222197532653809, 1.20772123336792, 1.3513752222061157, 0.5344645380973816, 1.883522629737854, 1.33008873462677, -0.17006517946720123, 0.7728874683380127, 1.3816072940826416, -0.08498707413673401, 0.05998168885707855, 0.2661784887313843, -1.377315640449524, 0.6193367838859558, 1.9709084033966064, -1.3222949504852295, 0.7429605722427368, 0.34033501148223877, -0.3545667231082916, 0.870789110660553, 0.6353029012680054, -0.6811888813972473, 0.5841502547264099, 0.8724050521850586, -0.2938593626022339, -0.2082863748073578, -0.611305832862854, -1.8971917629241943, -0.48934996128082275, -0.016170388087630272, -0.7431758642196655, -0.044637709856033325, -0.5427746772766113, 0.7256481647491455, 1.7229009866714478, 0.10254473239183426, -2.200277090072632, 0.6520141959190369, 1.165478229522705, 0.23873256146907806, -2.384657859802246, 0.23383022844791412, 0.7189283967018127, -0.30898672342300415, -0.08997274190187454, 0.8139485120773315, -0.2724558115005493, 0.8605202436447144, 1.0806750059127808, 0.6702581644058228, -0.8151845932006836, -0.4805953800678253, -1.6618658304214478, 1.039141297340393, 0.49037203192710876, -0.7705827355384827, 0.43733450770378113, -0.2806328237056732, -1.604526400566101, 1.6200063228607178], [0.6720248460769653, -0.9059661626815796, -2.298452615737915, -0.8405579924583435, 0.7679364085197449, -0.8902212977409363, -0.28355881571769714, 0.31762853264808655, -0.9180610179901123, 2.029313087463379, -1.21305513381958, 0.058426499366760254, -0.125694140791893, 0.3369656801223755, -1.3575623035430908, -1.8460932970046997, 1.0745384693145752, -1.7703685760498047, 0.000806718657258898, -0.9659406542778015, 0.6111741662025452, 1.0366365909576416, -0.01161919254809618, 0.1955510824918747, -0.17430172860622406, 0.19366507232189178, -1.3225494623184204, -0.6424829363822937, 1.1516293287277222, 1.6404790878295898, -1.5233864784240723, 0.19569216668605804, -0.39702391624450684, 0.6016295552253723, 0.3997529447078705, 1.7043057680130005, 0.7696222066879272, -0.9024313688278198, -0.912136971950531, 0.8004437685012817, -0.6850922703742981, 0.9256069660186768, 0.7152702212333679, 0.5110284686088562, -0.16835017502307892, 0.2591788172721863, 0.5533491969108582, 1.076319932937622, 0.3083622455596924, -0.5986992716789246, -0.7142749428749084, 0.8652678728103638, 1.4281277656555176, -1.5241913795471191, 0.4719833433628082, -0.0683407410979271, -0.8343325257301331, 0.021453654393553734, 0.2599850296974182, -0.5331273078918457, 0.4920637607574463, 0.5441380143165588, 0.5605444312095642, -0.5523408651351929, 0.422711580991745, 0.3045157194137573, -0.08046381175518036, -1.8856253623962402, -0.4556117653846741, 1.8451344966888428, -1.0677207708358765, 0.27890798449516296, -0.7884079813957214, 1.1965676546096802, 0.10028154402971268, -0.8054247498512268, 1.1125365495681763, -0.3239224851131439, -0.9841639399528503, 0.6766093969345093, -0.3889799416065216, -0.5818755030632019, -0.8813300132751465, 2.256072759628296, 0.7662956118583679, 0.6519956588745117, 0.6838708519935608, -0.4899952709674835, -0.07610573619604111, 1.12472403049469, -0.5279648303985596, -1.1244386434555054, -0.7431856393814087, 0.6638028025627136, 0.07159055769443512, -1.0823533535003662, -0.7514386177062988, -0.42523667216300964, 0.8970227837562561, -0.22705218195915222, -0.18150657415390015, -0.16294381022453308, -0.5002294182777405, -1.6076970100402832, 0.09814299643039703, 1.6700204610824585, -0.6599588990211487, 0.4245300889015198, -0.5802870988845825, 0.17302724719047546, 0.6430333852767944, 0.07296914607286453, -1.927393913269043, -2.1448488235473633, -1.047322392463684, -1.3902037143707275, 0.3742716908454895, 0.1484575718641281, -1.2462879419326782, 0.9400557279586792, 0.9171928763389587, 1.2361578941345215, 0.756033182144165, -1.0433882474899292, 1.7699795961380005, 1.6583136320114136, 1.1791119575500488, -0.6003215312957764, 1.7391135692596436, -0.6725979447364807, 0.7931826710700989, 0.57184898853302, -0.06232932209968567, -0.3952037990093231, -1.0978456735610962, 0.10977427661418915, -0.27725452184677124, -0.4120577871799469, 0.04687034338712692, 1.0912566184997559, 0.6450772285461426, 1.5416743755340576, -0.5768741965293884, 0.3742106556892395, -0.19853009283542633, 1.4621882438659668, 0.1165914535522461, -2.1661088466644287, 0.20384864509105682, -0.9222823977470398, -0.23096559941768646, -0.2598860263824463, 0.66402268409729, 0.163662850856781, 0.25892215967178345, 1.0562824010849, 0.3806554079055786, 0.2781068980693817, 1.8927093744277954, -0.3477837145328522, -0.9536716341972351, -0.796358585357666, -1.0794233083724976, 0.852422297000885, 0.7411240935325623, 0.349916011095047, -0.0632435530424118, 0.8758810758590698, 0.6379919052124023, -0.4628821313381195, -0.1855238527059555, 0.5372345447540283, 0.5135418772697449, -0.23263739049434662, -1.2985652685165405, -1.408046007156372, 0.9090783596038818, -1.2590227127075195, -0.46803346276283264, 0.4032852351665497, 0.9124913215637207, 0.8545743227005005, 0.48687660694122314, 0.750720739364624, 3.740389823913574, -0.9112332463264465, 0.27243438363075256, 0.8721226453781128, -1.718094825744629, 0.09372749924659729, -0.4393315017223358, -0.17680716514587402, -1.2352417707443237, -0.6697249412536621, -1.6224498748779297, 0.47090455889701843, -1.4521862268447876, 0.8450099229812622, -1.0376441478729248, -0.6210132241249084, -0.08970277011394501, 2.352808952331543, 1.051337480545044, -0.36767321825027466, 1.348371982574463, 0.30275648832321167, 0.5186745524406433, 0.35133907198905945, -0.8685344457626343, 0.6400547623634338, -0.6625755429267883, 0.10316195338964462, -0.5442366003990173, -1.1040704250335693, 0.4618377089500427, 0.39553409814834595, -0.07706934213638306, 0.12482563406229019, 0.21373848617076874, -0.9561423063278198, -1.817396640777588, 0.2586352229118347, -0.02455350011587143, -1.734761357307434, -0.3210235834121704, 0.44217896461486816, 0.14341917634010315, -0.40234172344207764, -0.9773565530776978, 0.2760913670063019, -0.8622781038284302, -0.659446120262146, 1.3314135074615479, 1.2569228410720825, 1.3828392028808594, -1.3451584577560425, -1.1687403917312622, 0.5649574995040894, 0.9430752396583557, -0.053230032324790955, 1.0204691886901855, 0.1310771107673645, 0.23256416618824005, 1.118788242340088, -0.17322614789009094, 1.0111089944839478, -2.296942949295044, 0.252278596162796, 0.6698225140571594, -0.6087918877601624, -0.4976048469543457, 0.8869577646255493, 0.5854452848434448, -0.4734188914299011, 0.6926758289337158, 0.48927268385887146, -1.552467703819275, 0.2918480932712555, 0.6836276054382324, 0.22753344476222992, 0.66314697265625, 0.39201483130455017, -0.7582516670227051, 0.7273668646812439, 0.09380955994129181, -1.3510810136795044, 1.975306510925293, -1.5147522687911987, -0.9372805953025818, 0.4737018346786499, 1.437532663345337, -0.35939428210258484, -0.2105128914117813, 1.7965103387832642, 0.4012426733970642, 0.08968029171228409, -1.3606518507003784, 0.223748117685318, -1.2261525392532349, 2.9192450046539307, -0.6681476831436157, -0.7866975665092468, 0.561578631401062, 1.5040085315704346, -0.6376383304595947, -1.0238826274871826, 0.6553551554679871, 0.6696342825889587, -0.2143525928258896, 1.5721189975738525, -0.31467142701148987, 1.8987233638763428, 0.6068780422210693, -2.1740376949310303, 0.8410661220550537, -0.3796883821487427, -1.691894769668579, 0.2197839617729187, 2.7819571495056152, -0.5285243988037109, -0.09030478447675705, -0.465472936630249, -1.0045312643051147, 0.8232018947601318, -0.5319386720657349, 0.9043782353401184, 1.131740927696228, 0.34492000937461853, 0.4280203878879547, 1.1203070878982544, -1.5097529888153076, -0.04876679927110672, -0.11924105882644653, -0.7742156386375427, -1.3040835857391357, -1.588313341140747, 1.0472955703735352, 0.2421361207962036, 1.4672479629516602, -0.38382527232170105, 0.6470728516578674, -0.07532468438148499, 0.6370819211006165, 0.3963429033756256, -0.09685568511486053, 0.0037202639505267143, 0.45498600602149963, 0.2679380476474762, -1.1752170324325562, 0.16800667345523834, 0.6138190031051636, -0.050132621079683304, 0.5349364876747131, -1.4300287961959839, -0.3543468415737152, 0.42667481303215027, -0.48781707882881165, -0.02569795772433281, 0.3296678066253662, 0.9941916465759277, 1.1039568185806274, -0.8724783062934875, -0.9034468531608582, 1.4931501150131226, -0.5731940269470215, -0.12790712714195251, -0.06677214056253433, 1.0350701808929443, -0.04622766748070717, 0.98270583152771, -0.5010838508605957, -0.08526716381311417, -1.3663471937179565, -0.02509130910038948, 2.0868446826934814, 0.5809154510498047, 0.08161679655313492, 0.178430438041687, 0.5329395532608032, -0.4627560079097748, -0.295344740152359, 1.197986125946045, 1.3014353513717651, 0.5979682803153992, -1.0382932424545288, -0.6044030785560608, 1.1202435493469238, -1.2813085317611694, -0.5110042691230774, -0.3860168159008026, -0.6007454991340637, 1.119497537612915, 0.6886163353919983, -0.4765743911266327, 0.8386154174804688, -1.3888345956802368, -0.6066001653671265, -1.086395263671875, -1.4171135425567627, -0.5289361476898193, -0.6463942527770996, 0.46215084195137024, 0.05797326937317848, 1.0997623205184937, 0.05195615440607071, 1.197312593460083, 1.603511095046997, 0.0004564513510558754, 0.28125348687171936, 0.7120053768157959, -0.4112779498100281, -0.682072103023529, 0.07741516083478928, 1.1983003616333008, -0.7769941091537476, 0.5524845123291016, -1.1912281513214111, 1.9112694263458252, -0.29848286509513855, -0.156387597322464, 0.1437283158302307, -0.8142757415771484, 0.6508035063743591, 2.795444965362549, -1.2184966802597046, -1.0803065299987793, -0.3704603314399719, 0.34184062480926514, -1.4773708581924438, -0.45899102091789246, 1.454943299293518, 0.613623321056366, -0.8930912613868713, -2.577572822570801, -0.8828068375587463, 0.00990211684256792, -1.4381461143493652, -0.312654584646225, -0.14448511600494385, 0.41743066906929016, -0.004113728646188974, 0.4550476670265198, -2.321268320083618, 2.209319591522217, -0.5053324103355408, -1.1851147413253784, -0.2324933111667633, 1.303930640220642, -0.46558594703674316, 0.5486319661140442, 0.06631210446357727, 1.6496583223342896, 0.3994539678096771, 0.019680993631482124, -1.0424705743789673, -1.698823094367981, -1.1179132461547852, 2.5012776851654053, -1.7854846715927124, -2.2795262336730957, -0.1015121266245842, -1.4368717670440674, 0.788690447807312, -0.981845498085022, 1.6634063720703125, 0.2282244712114334, -1.2465440034866333, 1.199166178703308, 0.29986560344696045, -3.0142579078674316, 0.08827697485685349, 2.225724458694458, -0.4590393900871277, 0.03359251841902733, 1.4030200242996216, -0.22822809219360352, -0.7591022849082947, -0.8286275267601013, -0.8392851948738098, 1.1017695665359497, -1.3498855829238892, -1.4970859289169312, -0.8138644099235535, 1.3765074014663696, 0.01775714010000229, 1.4708603620529175, 0.4157056510448456, -1.2676564455032349, 0.6279985904693604, 0.12980547547340393, -0.3077748715877533, 0.8375310301780701, 0.6680262684822083, -0.03503948077559471, 0.7474664449691772, -0.25602108240127563, 0.4359005391597748, 0.3530876338481903, 2.004202127456665, 1.0316776037216187, -0.21744826436042786, -1.4474207162857056, 2.7741870880126953, -0.3510940968990326, -0.10207705199718475, 1.3013410568237305, 0.36206740140914917, 1.162248134613037, 0.09320991486310959, 0.5203033685684204, 0.6573863625526428, -0.2176492065191269, 2.6380975246429443, 0.7009596228599548, 1.3352137804031372, -0.9488697052001953, -0.8835232853889465, -0.06486870348453522, -0.9804552793502808, 0.8346496224403381, -0.7384412884712219, 1.4585264921188354, 1.1535111665725708, -0.9321721196174622, -0.5756617784500122, -0.20209553837776184, 0.09117664396762848, 0.2730754017829895, 1.586190104484558, 1.6096251010894775, 1.1961705684661865, 1.1732754707336426, 0.10005497187376022, 1.3300775289535522, 0.24226132035255432, -2.2141273021698, -1.7913488149642944, 0.10596108436584473, 0.04173509031534195, 0.2040843516588211, -0.18934687972068787, -0.24652592837810516, 0.8850569725036621, 1.4249907732009888, -0.1506955325603485, -1.5125364065170288, -1.8166635036468506, 0.7188202142715454, 0.41419994831085205, 0.6023500561714172, -1.1286265850067139, 0.8352307677268982, 0.20768633484840393, -0.5182631611824036, 0.8036279678344727, 0.391924113035202, -0.5366107821464539, -0.31889206171035767, 1.0066362619400024, 0.5410515069961548, -0.15459606051445007, -0.04483815282583237, 2.734861373901367, -0.39853426814079285, 0.4716731905937195, -0.4424654245376587, 0.2628137469291687, 1.7082637548446655, 0.6805304884910583, 1.1027393341064453, 0.14697082340717316, 0.9807338714599609, -0.29396000504493713, -0.17519009113311768, -0.2376226782798767, 2.0325090885162354, 0.08206690847873688, 0.21114838123321533, 1.1686714887619019, 1.0965720415115356, -0.822990357875824, -1.1385130882263184, 1.9363038539886475, -0.9096595644950867, -0.4472290575504303, -0.43749740719795227, 0.6051872968673706, 0.8802631497383118, 0.4255233407020569, 1.6262627840042114, 1.1508433818817139, -0.2800804376602173, -0.5779376029968262, -0.1090456172823906, 1.0427173376083374, 0.5095217823982239, 1.378724455833435, -1.2730705738067627, -1.5581607818603516, -0.3304203450679779, 0.5093494653701782, 0.34730827808380127, 1.0707101821899414, 0.5416095852851868, -0.04583127796649933, -1.4488835334777832, -1.1739271879196167, -0.007111564744263887, -0.3893161118030548, 0.8984342217445374, -0.45334896445274353, 1.038114070892334, 0.8838480114936829, 0.8099245429039001, 1.1854901313781738, -2.9336540699005127, -0.3222184181213379, 0.6963380575180054, -0.7520927786827087, -0.0249779112637043, 0.8135260343551636, -0.2135889083147049, -1.4644217491149902, -0.26132503151893616, 0.13774561882019043, -0.6673870086669922, 0.3616463840007782, 0.276078462600708, -0.7591068148612976, -0.25477340817451477, 0.8571382164955139, -0.5582054853439331, -0.9929606318473816, 0.687665581703186, -0.8796655535697937, 1.3076037168502808, -0.048007283359766006, 0.5767057538032532, 2.6131346225738525, -0.3173910081386566, 1.3895058631896973, 0.6856622695922852, -0.7765572667121887, 0.1111396923661232, -0.24381671845912933, -0.5702961087226868, 0.580296516418457, 1.0300252437591553, 0.6720419526100159, 0.10573713481426239, -1.9148294925689697, 1.833020567893982, 1.650317668914795, -1.1695774793624878, 1.8397356271743774, 0.49350622296333313, 1.835507869720459, 1.6998108625411987, -0.16477204859256744, 0.43838050961494446, -0.9416755437850952, -0.25892895460128784, 0.7942925691604614, -1.7376383543014526, 1.9197022914886475, 0.36051973700523376, 0.2482374608516693, -2.0549986362457275, -0.675423800945282, -1.7259998321533203, 0.5255626440048218, 0.061369381844997406, -0.6855634450912476, -0.9162648916244507, -0.6962330341339111, -0.020784474909305573, -1.2697649002075195, -0.5803132653236389, -0.11536211520433426, 0.3320920467376709, -1.5791027545928955, 0.2256750911474228, -0.10893520712852478, 0.5863217115402222, 0.5941857099533081, 0.40015318989753723, 1.589362621307373, -0.9959452748298645, -1.064829707145691, -0.5771906971931458, 1.1772202253341675, -0.4774312376976013, -0.1813594251871109, 1.4959636926651, 0.7870500683784485, 0.7280343174934387, -1.1693798303604126, 1.9198752641677856, 0.0858636200428009, 1.5462157726287842, -1.4914389848709106, 1.7259480953216553, 0.4123338758945465, -0.420230895280838, -1.512900710105896, 1.046644926071167, 0.23660428822040558, -0.6174725890159607, -0.5994341969490051, -1.4484400749206543, -0.6311504244804382, -0.2666926681995392, -0.6582289338111877, -1.0622259378433228, 0.12477155029773712, -1.1282559633255005, -0.18046768009662628, 0.24361169338226318, -1.0058668851852417, 1.2497715950012207, 0.5102825164794922, 0.8333441019058228, 0.13074517250061035, -2.1381003856658936, 0.5514584183692932, 0.4462571442127228, 0.7727651000022888, 1.0912106037139893, 1.7795484066009521, 1.044144630432129, -0.46572113037109375, -0.7485638856887817, -0.34714093804359436, 1.745624303817749, 0.4128091335296631, 2.097489356994629, 0.1693788766860962, -0.6904016733169556, -0.5350518226623535, -0.513166606426239, 0.1474440097808838, -0.8832635283470154, 0.38039347529411316, 0.7018084526062012, 1.7555283308029175, -1.548824667930603, 0.6027332544326782, 1.1501526832580566, -1.3173785209655762, 1.4665710926055908, 1.7974250316619873, -0.27825918793678284, 0.36823052167892456, 0.09109807014465332, -0.8104979991912842, 2.1458752155303955, 0.07883717119693756, 0.8932796716690063, -0.2854821979999542, -0.056017473340034485, -1.4427614212036133, 0.7930928468704224, -0.7631059288978577, -1.1595187187194824, -1.3071640729904175, -1.484702229499817, 0.18237988650798798, 0.3626602590084076, 0.24424827098846436, 0.42250174283981323, -0.7581759691238403, -2.8699607849121094, -0.731204628944397, -0.7070012092590332, -1.3365246057510376, -0.8357508182525635, -1.5271337032318115, 0.1961991935968399, 0.5269683599472046, -0.7961316704750061, -0.7289004325866699, 1.4900273084640503, -0.8901973962783813, -0.07207057625055313, -1.0922280550003052, 0.1689135730266571, 1.629106044769287, -0.42030662298202515, 0.3442394435405731, 1.2958430051803589, 0.25757312774658203, 0.047515835613012314, 0.42499756813049316, -0.03451526165008545, 0.8658052086830139, -1.2769665718078613, 0.7892969846725464, -1.435516357421875, -0.09890739619731903, -0.07217474281787872, 1.035765528678894, 0.4439695477485657, -0.34075358510017395, -0.2259880006313324, 0.0886145606637001, 1.1205823421478271, 0.10075949877500534, -0.776248574256897, -1.878511905670166, -0.7947940230369568, 1.3627182245254517, 0.6865910291671753, -0.4620252847671509, 1.5687534809112549, -1.0346652269363403, 0.8583673238754272, -0.6392729878425598, 0.9457510709762573, -0.09249276667833328, -0.9905956387519836, 0.3039708733558655, 2.1455202102661133, 0.5286272764205933, -0.5479868054389954, 0.25417086482048035, -1.3676602840423584, 0.5382819771766663, -0.6359077095985413, -1.405300259590149, 0.28776147961616516, -0.44419971108436584, -1.7381616830825806, -1.967741847038269, -0.007660670671612024, -0.5327259302139282, 0.205247700214386, 0.1377701461315155, -0.2848597466945648, 0.5192850828170776, 1.34663724899292, -1.3138235807418823, 0.44963255524635315, 0.058846067637205124, -0.29528626799583435, 0.36653879284858704, 1.5916742086410522, -0.14235423505306244, -1.1179656982421875, -0.8690912127494812, 1.3474963903427124, -0.15721212327480316, -0.08899971842765808, -1.5930029153823853, -0.5807406306266785, -0.603566586971283, -2.1359963417053223, 0.5653009414672852, 1.6181509494781494, 0.3638252913951874, -0.7748825550079346, 1.9527649879455566, 0.06047213077545166, 0.13927729427814484, -0.13481773436069489, 0.463316947221756, 1.3932546377182007, 0.07815980166196823, -0.6031153798103333, 0.5415546298027039, -0.22766146063804626, 0.8718230724334717, -0.886555016040802, -0.10290287435054779, 0.23452794551849365, -1.2393879890441895, 0.006596149876713753, 0.15477007627487183, 0.2293826937675476, 1.149104118347168, 1.3863730430603027, 0.6951556205749512, 0.3526003956794739, -0.24758493900299072, -0.34142163395881653, -0.6029453873634338, -0.7324762344360352, 0.3709583580493927, -1.1296300888061523, 1.4072585105895996, -1.6737033128738403, -0.05426649749279022, -0.2887061536312103, 0.7124140858650208, 0.9570600986480713, 0.9591695070266724, -2.370147943496704, 0.7648637890815735, -0.8739809393882751, 0.16242629289627075, 0.08458694815635681, 1.8362709283828735, 1.4980069398880005, -0.31780901551246643, 1.3121912479400635, 0.8275041580200195, -1.0378319025039673, -1.040496587753296, -0.7861359715461731, 0.9669580459594727, 0.5420283675193787, -2.3771889209747314, 0.22348544001579285, -1.2540972232818604, 1.048404335975647, -0.5336141586303711, -0.6086535453796387, 0.11875554174184799, -1.5815839767456055, -0.42697393894195557, -1.4756293296813965, 1.639218807220459, 0.5415052771568298, -1.5115060806274414, 1.756552815437317, 0.6063722372055054, 0.39325544238090515, -1.0444778203964233, 1.0164624452590942, 0.8179699182510376, 1.8304113149642944, -0.5621888637542725, 0.4045814573764801, -0.5460256338119507, -0.7294022440910339, -1.6332008838653564, 0.5437144637107849, -0.5064557790756226, -2.0383636951446533, -2.221296787261963, -0.7133330702781677, 0.23332472145557404, 0.8538421988487244, -1.4211530685424805, 1.0274944305419922, -1.1523733139038086, 1.5569589138031006, -0.8882806301116943, 0.8742496967315674, -2.570003032684326, 0.9903207421302795, -0.7075499296188354, -0.6733144521713257, -0.6840038895606995, 1.6824514865875244, 1.0524401664733887, 0.029366973787546158, 0.9652538895606995, 0.5292870998382568, -1.4197205305099487, 0.2530103027820587, 1.8943179845809937, 0.07309779524803162, 0.03628256916999817, 0.33756041526794434, 0.980619490146637, 1.2818069458007812, 1.8740099668502808, 0.8846839666366577, -0.9722720980644226, -0.7014085650444031, 1.1772677898406982, -0.07304670661687851, 0.8503620624542236, -0.8811768293380737, -0.6359725594520569, -0.5559191703796387, 0.7567366361618042, 0.04313718527555466, 0.06377250701189041, -0.8159045577049255, -1.3800023794174194, 1.5659593343734741, -0.7460805773735046, -0.13350588083267212, 0.34452682733535767, -0.39639684557914734, -0.527044415473938, -1.5215978622436523, -1.6354360580444336, 0.4696004092693329, 0.31823280453681946, -0.5198339819908142, 0.8313369750976562, -0.5687013864517212, -0.24527107179164886, -1.56233811378479, 0.6677805781364441, 0.6141285300254822, 0.20240247249603271, -0.08085491508245468, 0.5582959055900574, -0.007368676830083132, -1.2279456853866577, 1.2197070121765137, 0.4910718500614166, -0.05349203944206238, 0.8820828199386597, 2.2436037063598633, -0.3684452772140503, -0.6615138053894043, -0.3015718162059784, -1.4293493032455444, 0.8580412864685059, 0.21548372507095337, 0.6394438743591309, 0.11946393549442291, 1.4818229675292969, 1.6098637580871582, -0.6655281782150269, 0.1593708097934723, -0.2854367196559906, 1.552789568901062, -1.1169369220733643, -0.9725863933563232, -0.16909179091453552, 0.6012950539588928, -0.21615295112133026, 0.6969037055969238, -1.306814432144165, 1.6445177793502808, -0.46597200632095337, -0.9304797649383545, -0.16535572707653046, 0.17742983996868134, -1.3523403406143188, -0.9400591254234314, -0.7066237926483154, 0.5599998235702515, -1.6304618120193481, 1.5750356912612915, -1.6141390800476074, 2.480027675628662, -2.6875243186950684, 0.9602569341659546], [-0.20826177299022675, -1.0629949569702148, 0.07853912562131882, -0.22824300825595856, 0.4712393879890442, 0.8474246263504028, 0.7817564010620117, 0.16434352099895477, 1.2552295923233032, -1.550861120223999, 0.41313740611076355, 0.5828738212585449, -2.40641713142395, 0.08135902881622314, -2.000199794769287, 0.8659494519233704, -0.0637437179684639, 0.45787638425827026, 0.9013516306877136, -1.4468379020690918, 0.2727624475955963, 0.3865337073802948, -1.0422505140304565, -0.2225024551153183, -2.0157549381256104, -0.01979837380349636, 0.7188970446586609, -1.1184149980545044, -0.8342136144638062, -0.9145391583442688, -0.43265634775161743, 0.2644829750061035, -0.5211458206176758, -0.12667594850063324, -0.5655381679534912, -0.7941001653671265, 0.303088903427124, 0.9716386198997498, -1.796047568321228, 2.7170591354370117, 0.34056171774864197, -0.2656554877758026, -0.7766964435577393, 1.1885507106781006, -1.277135968208313, -0.16361674666404724, -0.7564550042152405, 1.9053759574890137, -1.8455578088760376, -1.2123521566390991, -1.3182860612869263, -1.3337924480438232, -0.035924751311540604, 0.28147611021995544, -0.21900701522827148, 1.389274001121521, 0.1837945133447647, 1.615739107131958, 0.6409438252449036, -0.36167091131210327, 0.9332603812217712, -1.9395122528076172, 0.19456793367862701, 0.7987201809883118, -0.2870812714099884, -0.8976407051086426, -1.348048210144043, -0.1331276148557663, -0.3925182819366455, -0.1526380479335785, 0.7553017735481262, -1.0519403219223022, 0.6951354742050171, 2.1244077682495117, -0.048558931797742844, -0.6829445958137512, -0.5383845567703247, -1.8810724020004272, 0.07285258173942566, -0.8391999006271362, 1.2320642471313477, 0.03413555398583412, -0.3809579908847809, -0.9194852113723755, -1.597780704498291, 0.5792737603187561, 0.4732780456542969, -0.7492181658744812, -1.7239148616790771, -0.19882513582706451, -0.7060621380805969, -0.6851168274879456, 0.8812355995178223, 1.9216457605361938, -2.680129289627075, -1.516722559928894, 2.917952537536621, -1.0356343984603882, 1.0941375494003296, -0.23466038703918457, 1.2849818468093872, -1.7826488018035889, -1.4653738737106323, 1.4377257823944092, -0.7751206159591675, 1.138907790184021, -0.9106515645980835, 1.0328675508499146, -0.507756769657135, 1.1665382385253906, -1.6480156183242798, 0.3491005003452301, 0.35864710807800293, 0.5784624218940735, -0.7418320775032043, 0.07647669315338135, -0.7956850528717041, 0.3898107409477234, -0.9531291723251343, -0.013932821340858936, 1.1599754095077515, -0.26454395055770874, 0.16607287526130676, 0.8975415229797363, -0.5420810580253601, 0.6445624828338623, -0.2086840718984604, -0.5465349555015564, 0.14857396483421326, -0.5072641968727112, -0.6452509760856628, 0.39871394634246826, -0.28511157631874084, 1.1676651239395142, 1.3198059797286987, 0.5053158402442932, -0.30891796946525574, -0.2222091555595398, 1.7378937005996704, -0.09932266175746918, 0.018070990219712257, -1.72613525390625, 0.38659968972206116, 0.14585523307323456, 0.5796996355056763, -0.664380669593811, 1.3958938121795654, 0.7143662571907043, -0.3070623278617859, -0.5908028483390808, 1.5522757768630981, 1.4697836637496948, -0.36959847807884216, 0.4081163704395294, 0.7238074541091919, -0.6859180927276611, 1.192613959312439, 1.080319881439209, -1.0416072607040405, -0.9246666431427002, 2.6513094902038574, 0.6247915029525757, -1.0290236473083496, 0.34282317757606506, 1.3890100717544556, 1.3963580131530762, 1.1026228666305542, -0.31096404790878296, 0.6883952617645264, -0.23324023187160492, -0.14475397765636444, -1.0153449773788452, 1.3753777742385864, 0.1027691587805748, 0.10103874653577805, -0.21883554756641388, -1.2206015586853027, -0.18783719837665558, -0.0629412978887558, 0.9767402410507202, -0.3165145218372345, 1.443166732788086, 0.38780468702316284, 0.7772902846336365, -0.28049057722091675, 0.3762311041355133, 1.4829761981964111, -1.57992422580719, 1.1218549013137817, -0.5997183322906494, -0.7888861894607544, 0.2949107885360718, -1.7718865871429443, 1.0840612649917603, -0.22892452776432037, 0.3648105263710022, 0.6410521268844604, -0.22337140142917633, -0.39422717690467834, 1.341557264328003, 0.3082138001918793, 1.5791611671447754, -0.037066418677568436, 0.5355024933815002, 1.0099717378616333, -0.9952132701873779, 0.2505972385406494, -0.646248996257782, -0.7804862856864929, -0.7578693628311157, -1.2627452611923218, -1.155258297920227, -1.0177037715911865, -0.8411076664924622, 1.6696326732635498, 1.184857726097107, -0.5493858456611633, 0.9935132265090942, 0.499843955039978, 1.2256309986114502, -2.061403512954712, 1.440457820892334, -0.03444977104663849, 0.8593666553497314, -1.037775993347168, 1.4370030164718628, -1.2779059410095215, 0.17107073962688446, -1.2696382999420166, 0.39195793867111206, 0.7510350942611694, -0.49783697724342346, -0.9644723534584045, -0.4873619079589844, -1.8432872295379639, 0.25582265853881836, 0.8606320023536682, -0.15884457528591156, -0.011122000403702259, 0.11033833771944046, 0.11902880668640137, 0.390423983335495, -1.4131673574447632, -0.003734692232683301, 0.29177379608154297, -1.4543635845184326, -0.9965361952781677, 0.27302780747413635, -1.3867859840393066, 0.33041051030158997, -1.2753047943115234, -0.44971442222595215, -1.1313055753707886, 0.003196683479472995, -0.8133281469345093, -2.069584608078003, -0.12221997231245041, -1.827314853668213, -0.07498742640018463, 0.48904743790626526, 2.054792642593384, 0.8432092666625977, -0.6579044461250305, -1.0148359537124634, -0.11979412287473679, -0.6511297225952148, 0.8230613470077515, 0.4141581058502197, 0.8909414410591125, -0.4268573820590973, 1.0320930480957031, -0.746777355670929, 0.5201743245124817, 1.6962981224060059, -1.2721731662750244, -0.29353466629981995, 0.3017185926437378, -0.41511183977127075, -1.6517517566680908, -0.5177057385444641, -1.0949351787567139, -1.5125516653060913, 0.1624966859817505, 0.6946436166763306, -1.019495964050293, -0.5402865409851074, 1.092702031135559, 0.04339565336704254, -0.47887566685676575, 2.330349922180176, -0.7995911836624146, -0.05536947026848793, -0.0851312130689621, -1.1006076335906982, -1.7571699619293213, -0.17615319788455963, -0.2668432891368866, -0.030661610886454582, 1.1343122720718384, -2.1906909942626953, 0.971875786781311, -0.38453376293182373, 0.5712890625, -1.9562944173812866, 3.394150733947754, -0.6476271748542786, 0.23680523037910461, 0.4565148949623108, -0.6241559386253357, 0.6064014434814453, -0.41413336992263794, -0.32716530561447144, -0.47855669260025024, 0.8178806900978088, 1.3108075857162476, 0.9426842927932739, 0.015507197007536888, -0.5346518754959106, -0.05118340253829956, 1.343775749206543, 1.3127516508102417, -0.23803383111953735, -0.8784793615341187, 0.7952513694763184, 0.42280879616737366, -0.43404245376586914, 1.6242711544036865, 0.6616778373718262, -1.2633837461471558, 0.5816627144813538, -0.5293464064598083, 0.740896463394165, 0.37441766262054443, -0.19494864344596863, -0.27115002274513245, 0.8737853169441223, 1.5114810466766357, 2.203993797302246, 0.11153983324766159, -0.9603530168533325, -0.5287938714027405, -0.40115922689437866, 0.21496155858039856, 0.5613909363746643, -1.0474433898925781, -0.9189339280128479, -0.14847157895565033, 0.2689557373523712, 0.32550403475761414, 0.6685736179351807, 0.6040041446685791, 0.27452030777931213, -0.2749975919723511, 1.6872681379318237, -2.5040273666381836, 0.7963026762008667, 1.7167304754257202, -0.21999621391296387, -0.18386738002300262, -0.5880654454231262, -0.3854234516620636, 0.5851994752883911, -0.35784003138542175, 0.4900124967098236, 0.15989156067371368, -0.7493316531181335, -1.0055161714553833, -0.5475399494171143, -0.9108626842498779, 0.8225623965263367, 1.7603132724761963, 0.4805765748023987, -0.4873217046260834, -0.0994369238615036, 0.8008415102958679, 1.0733071565628052, -0.5433143973350525, 0.9279357194900513, 1.1796882152557373, 0.0031769126653671265, 2.233654260635376, 1.6178830862045288, -0.9581446051597595, -0.8821898102760315, -2.144243001937866, -1.3077352046966553, 1.8719148635864258, 1.4743560552597046, 0.2503972053527832, 0.9164602756500244, 0.06691958755254745, 0.11093238741159439, 0.4251735210418701, 1.790889859199524, 0.9774389266967773, -1.7194074392318726, 0.08395085483789444, -0.7037821412086487, -1.6353527307510376, -0.4985966980457306, -0.31531018018722534, -0.5072093605995178, -1.5146313905715942, 0.6569423675537109, 0.4873577058315277, -0.3503582179546356, 0.3585507571697235, 0.5645508766174316, 1.3685206174850464, 0.34066328406333923, -0.43708258867263794, -0.3639383912086487, -0.08686437457799911, -1.0333335399627686, 0.05834805220365524, -0.20579911768436432, -0.6869710087776184, -0.6048205494880676, 1.7003040313720703, 0.29336223006248474, 0.25494664907455444, 0.4921956956386566, 2.128990888595581, -0.14257366955280304, -1.2878265380859375, 0.48913145065307617, 1.9812829494476318, 2.5550942420959473, -0.8730084300041199, 1.2672079801559448, 0.3542822301387787, -1.0616035461425781, -0.6294061541557312, 0.09837649762630463, 1.0681520700454712, -0.2569558024406433, 1.7480368614196777, 0.584954023361206, 0.04550152271986008, -0.2176787406206131, 0.22477547824382782, 2.0562329292297363, 0.5575007796287537, -1.9843907356262207, 0.9351779222488403, -0.07777830958366394, 0.6901222467422485, -0.25340771675109863, 0.1340402513742447, 0.9461541771888733, -0.6551306843757629, 0.2333846092224121, 0.9581405520439148, -0.13937219977378845, -0.02877681888639927, -1.3282580375671387, 3.324324369430542, -0.09479805827140808, -0.7685038447380066, -0.43348053097724915, 3.152111053466797, -0.15534257888793945, 0.8701711297035217, 1.372274398803711, -0.3700438439846039, -0.7070888876914978, -2.247720241546631, -0.1557757407426834, -1.354414939880371, -1.4235577583312988, -0.7888240814208984, 0.11197920143604279, -0.07652535289525986, 0.29426270723342896, -0.6325668692588806, -2.2063474655151367, 0.5408477783203125, -1.7412323951721191, -1.3891593217849731, -0.48175638914108276, 1.4954222440719604, -0.47466108202934265, 0.1692543774843216, 1.3109346628189087, 0.5783042907714844, -1.0178364515304565, -0.4388540983200073, 1.9248813390731812, -0.6945866346359253, 0.6687402725219727, -0.1362856775522232, -1.4968730211257935, -1.225427508354187, 0.39303353428840637, 0.25453850626945496, 1.4704328775405884, -1.0881540775299072, -0.5053671598434448, -0.9337979555130005, 1.6052794456481934, -2.3644485473632812, -0.7307818531990051, 0.03610911965370178, -1.2651307582855225, 1.6004605293273926, 0.9764246344566345, -0.5780631899833679, 0.3358434736728668, -0.25914594531059265, -0.4428369104862213, -0.3237988352775574, 0.2303684502840042, -0.9207867383956909, -0.46989011764526367, 1.6791155338287354, -1.2081063985824585, -0.4847157597541809, 1.0129538774490356, -1.12163507938385, -1.4915269613265991, 0.6686100959777832, 1.930310845375061, 0.9371894001960754, 0.2550811171531677, 0.5884027481079102, -0.17325547337532043, 0.055589474737644196, 1.0238159894943237, -0.1661255657672882, -1.0170814990997314, 1.1152561902999878, 2.711656093597412, -0.7429015040397644, -0.30504110455513, -1.111228108406067, 1.1991605758666992, -0.38582244515419006, 0.5975764989852905, -0.18344934284687042, -1.6027249097824097, 0.7351555824279785, -0.748325765132904, -2.5388572216033936, -0.29127612709999084, -0.16472116112709045, 0.027277600020170212, -0.9969309568405151, -0.653994083404541, 0.5117005109786987, -2.2848355770111084, -1.6071571111679077, 0.4858482778072357, 2.252610921859741, -1.3229939937591553, -0.9417235255241394, -0.9208017587661743, -0.9513391852378845, -0.742296040058136, 0.6054611802101135, -1.3723900318145752, 0.3868104815483093, -1.8263123035430908, -0.7028002142906189, -0.5511690378189087, 0.09689969569444656, -0.5569943189620972, -0.3504130244255066, -0.3548562228679657, -1.393908143043518, -0.8043234348297119, -0.6399067044258118, -1.5541119575500488, -2.1761019229888916, 0.20749235153198242, 0.20467416942119598, 0.2708056569099426, -0.6602237224578857, 0.21514038741588593, -0.824011504650116, -0.021639063954353333, 1.2309114933013916, -0.2239539921283722, 0.8700281977653503, 0.3700622618198395, 0.32962241768836975, 2.375808000564575, -0.6030237674713135, -0.5264683365821838, -1.6762114763259888, -0.5648050904273987, 0.2843649983406067, 1.0336834192276, -1.0646462440490723, -0.0835660919547081, 0.6063541769981384, 0.31976833939552307, 1.13738214969635, 0.10638061165809631, 1.2577178478240967, 1.7689497470855713, 1.5802799463272095, 1.3686343431472778, -0.42579391598701477, 0.19114820659160614, -0.7420684099197388, -0.80292809009552, -0.40327131748199463, 0.01854262314736843, -0.7792083621025085, 1.2774529457092285, 1.4006778001785278, -2.2366373538970947, -0.33556726574897766, -0.5463187098503113, -0.07217685133218765, 1.3439291715621948, -0.36205101013183594, 0.9134500026702881, 1.240519642829895, -0.05281786993145943, -0.37786710262298584, 0.19712042808532715, 1.632195234298706, 0.6775897145271301, -0.34211769700050354, -0.45556357502937317, -1.0613168478012085, 1.0296655893325806, -1.352696418762207, -0.8217650055885315, 0.730049192905426, 0.9493504762649536, -0.8290714621543884, 1.21938955783844, -1.096876859664917, 0.493303507566452, -0.09348320215940475, 0.0955573096871376, -1.0317373275756836, 0.6392924189567566, -0.8947627544403076, 0.5314067602157593, -0.34402886033058167, -0.4566442668437958, 0.18334724009037018, 0.20253147184848785, -0.603029191493988, -0.6150754690170288, -2.0653789043426514, 1.7681609392166138, 0.6171281337738037, -0.10062139481306076, -0.7948310971260071, -0.9817976951599121, 0.15680789947509766, -1.590247392654419, -1.7998992204666138, -1.9689139127731323, -2.3120930194854736, -1.0007953643798828, 0.9920924305915833, 2.295095682144165, 0.9793188571929932, 0.683634877204895, -0.12447655200958252, -1.0755387544631958, -0.84478759765625, 0.30058524012565613, 1.0789942741394043, -1.6610440015792847, 0.8075026869773865, -0.5766379833221436, -0.9984820485115051, 0.35721346735954285, -1.2787129878997803, -0.5179610848426819, 0.37283915281295776, 2.5519185066223145, -0.5092913508415222, 0.04907144606113434, 0.5758445858955383, 1.1337571144104004, 0.6345552802085876, 1.205366849899292, 1.0402930974960327, 0.1566716730594635, 0.03810003027319908, -0.46686622500419617, -0.6646838784217834, -0.532439649105072, 0.684482753276825, 1.3235903978347778, 0.06048377603292465, 1.2271226644515991, 1.6965816020965576, -0.6890259385108948, -0.8545538187026978, -0.20363032817840576, 0.07232888042926788, -0.8498229384422302, -0.9233992695808411, 0.809908926486969, 0.898605465888977, 0.9127991199493408, 0.9813461899757385, 0.14617909491062164, 0.2192721664905548, -0.4794962406158447, 0.44462132453918457, -1.4521530866622925, -1.3918898105621338, -0.5279064774513245, -1.0905534029006958, -3.076427698135376, 0.2783292233943939, -0.1026126965880394, 0.295424222946167, 0.9314973950386047, -0.4811035096645355, -0.8206153512001038, 0.08633061498403549, 0.22056716680526733, 0.1986200213432312, 1.017311930656433, -0.7625444531440735, 0.8809241056442261, 0.7977948784828186, -0.7305637001991272, -0.9055975079536438, 1.3756685256958008, 1.3343359231948853, 0.9334836602210999, -1.7071820497512817, 0.5794039368629456, -1.1480319499969482, -0.3719381093978882, 0.48885709047317505, -1.9455385208129883, -0.6927034258842468, 0.6249198913574219, 0.4640466272830963, -0.2907567620277405, 0.20620781183242798, -1.257582664489746, -1.1366227865219116, -0.17879857122898102, -0.3847143054008484, 0.2904036343097687, -0.7604588270187378, -1.1141680479049683, 1.7054909467697144, -0.5870105028152466, -1.1917123794555664, 1.3696483373641968, 0.14941222965717316, -0.953125536441803, -1.589821696281433, -0.3472428619861603, 0.1408572942018509, 1.2077077627182007, -0.20276105403900146, 1.4235161542892456, -0.8272618055343628, -1.8702503442764282, -1.0584261417388916, -0.5297549366950989, -0.4235111474990845, 0.35852885246276855, 0.2083148956298828, 0.34342533349990845, -0.3755693733692169, 0.032783642411231995, 0.21191605925559998, 0.9155148863792419, -0.4805850088596344, 1.256972074508667, 0.3454250991344452, 2.1164567470550537, 0.14293640851974487, -1.526082992553711, 0.007281463593244553, -0.540899395942688, 0.08072970062494278, 0.8309959769248962, -1.5076476335525513, 1.946882963180542, 0.6761036515235901, -1.2207716703414917, -1.186461091041565, -0.31884992122650146, 0.7940242290496826, 0.5140444040298462, -0.2857699990272522, -0.28498736023902893, 1.01836097240448, 1.9109524488449097, -0.38730308413505554, -0.9881547093391418, -0.196369007229805, 1.00990891456604, 0.27634990215301514, 1.3396152257919312, 0.668016791343689, -0.17566536366939545, 1.052872896194458, -0.4782560169696808, 1.0095977783203125, -0.515515148639679, -0.047583408653736115, -1.4497660398483276, -0.6588596701622009, 0.660356342792511, 0.03398386016488075, 0.10241860896348953, -0.08560826629400253, 0.9374997615814209, -0.22308501601219177, -1.6870721578598022, 0.3105383515357971, -0.31227952241897583, 1.290649175643921, 0.1244446262717247, -1.111063003540039, -0.5965516567230225, -0.9563606977462769, -2.054786443710327, -0.875300943851471, 0.3540612757205963, -0.014579718932509422, -1.1040823459625244, 0.368134081363678, -0.5855420827865601, -0.25018447637557983, -0.2558676600456238, 0.1599702686071396, -1.8109707832336426, 1.314934492111206, -0.45189517736434937, -0.10268373042345047, -0.7455102801322937, -0.6440834999084473, -0.4330446422100067, 0.549098014831543, 0.5402624011039734, -0.7853624224662781, 0.10032700002193451, 1.1315242052078247, 0.29618018865585327, -0.08507932722568512, -0.02467227168381214, -0.30225127935409546, 0.5748569965362549, 0.7020758986473083, 0.48548856377601624, 0.17616719007492065, 0.5508878231048584, -0.0098505150526762, -0.8327143788337708, 0.3359850347042084, -0.3416844606399536, 0.7504453659057617, 0.8476526737213135, -0.8537596464157104, -0.34004488587379456, 0.6532821655273438, -1.216392159461975, -0.8980509638786316, 0.7985009551048279, -0.5115050077438354, -0.3774988055229187, -0.4990955591201782, -0.4076840579509735, 0.633504331111908, -1.7947583198547363, -0.7206019759178162, 1.235348105430603, 0.19038306176662445, -0.6008495688438416, -1.3934487104415894, -1.247431993484497, -0.8810226917266846, -1.4152462482452393, 0.21770784258842468, -1.4962371587753296, 0.1267601102590561, 0.35662779211997986, -1.3165593147277832, -0.5656066536903381, -0.07684999704360962, 0.9304993748664856, 1.1529561281204224, 0.41548579931259155, -1.6418544054031372, 0.6307586431503296, -0.38633593916893005, -1.2990113496780396, -0.5887039303779602, 0.7899627089500427, -0.8731316924095154, -0.8768026232719421, -0.9921765327453613, -0.4839775562286377, -0.06377756595611572, -1.364033579826355, 0.6652851104736328, -1.1458375453948975, -0.6424735188484192, 0.5773855447769165, 0.6880277395248413, -0.19724532961845398, 1.4977753162384033, -0.6964077949523926, 0.648916482925415, -0.08928826451301575, -0.5701409578323364, 0.2968834340572357, 0.19383612275123596, 0.4671647846698761, -0.9961804747581482, 1.1005895137786865, 0.2924213409423828, -0.24538420140743256, 0.32108548283576965, -0.6707640290260315, -0.9858522415161133, 0.40724658966064453, 0.5310210585594177, 0.09408418834209442, 0.025111956521868706, -0.8525756001472473, 0.026489926502108574, 0.2038303017616272, -1.022362470626831, 0.42739802598953247, -0.193559929728508, -0.16571994125843048, 2.2013399600982666, -1.6099791526794434, 0.8949553966522217, 0.05295581370592117, 0.7005317807197571, -0.5721486806869507, 1.2773182392120361, -0.694823682308197, -0.32669153809547424, -0.3852665424346924, -0.945803165435791, 1.4930086135864258, -0.7442225217819214, -0.06678207963705063, -1.8225446939468384, 0.9265801906585693, -0.25690025091171265, -0.6332796216011047, -0.966331422328949, 0.5481058359146118, 1.4549044370651245, 0.6766175031661987, 0.35071876645088196, 0.475252240896225, -0.49237650632858276, 2.5824053287506104, -0.48539718985557556, -0.6657012104988098, 0.5403790473937988, 1.2223176956176758, 1.114753246307373, 0.7063162326812744, 0.7144607901573181, 0.04406360164284706, -2.358520030975342, 0.3417634963989258, 0.6833488345146179, -1.1521551609039307, 0.10089527070522308, 0.3671809732913971, 1.000569462776184, -1.3247530460357666, -0.4367135167121887, 1.5351074934005737, 0.1998477727174759, -0.8888599276542664, -0.4047575891017914, 0.2613452672958374, 0.9940432906150818, 0.0013685455778613687, 2.80484938621521, 0.4574939012527466, 0.16560469567775726, 0.09790129214525223, -0.18270891904830933, 0.7202867865562439, -0.6292409300804138, -0.46871304512023926, -0.25943803787231445, 0.02586127258837223, 0.0011376733891665936, -0.46833041310310364, 0.28224754333496094, -2.5129482746124268, -0.34329259395599365, 0.3575620949268341, 0.586787760257721, 1.0456138849258423, 0.7053006291389465, 0.23881575465202332, 0.11449135094881058, 1.5583624839782715, -1.5536437034606934, 0.517918586730957, 0.48205071687698364, 0.11106780916452408, 0.04342697933316231, 0.13371235132217407, 0.1700274646282196, 1.2773678302764893, 0.9200337529182434, 1.3500361442565918, -0.3054817318916321, -0.750236988067627, 1.2842262983322144, 0.7607027292251587, 0.24266545474529266, 0.2796962857246399], [1.5798940658569336, 0.524945080280304, 0.016382908448576927, -1.0870347023010254, -0.004811557475477457, -0.8047865629196167, -0.1549186259508133, 1.7767237424850464, 1.9435410499572754, -0.31668221950531006, -0.2868529260158539, -1.4792299270629883, 0.6374737620353699, 0.8443831205368042, -1.0963371992111206, -0.4843187928199768, 1.2925546169281006, 0.5567014813423157, -1.3804901838302612, 0.7675448656082153, -1.410510778427124, -0.35559287667274475, -0.3188878893852234, 1.0036412477493286, -0.43986886739730835, 0.3229919672012329, -0.7086628675460815, -0.273922324180603, -1.7282599210739136, -0.8152300715446472, -1.1449106931686401, -0.121008962392807, 0.7291242480278015, -0.7488828301429749, -0.5501826405525208, 0.9832612872123718, -0.16242709755897522, 0.9049610495567322, -1.556199550628662, 0.011697315610945225, -0.01901034638285637, -1.5066771507263184, -0.6474969387054443, 0.49571576714515686, -1.1818664073944092, -0.2384907454252243, 0.23350153863430023, -0.4374087452888489, -0.6157616972923279, -1.7056300640106201, -0.6401106715202332, -2.002511739730835, -0.11443420499563217, -0.21743226051330566, 0.702460765838623, 0.29075485467910767, 1.3859541416168213, -2.099025011062622, -0.08105643093585968, 0.7904324531555176, 2.8067786693573, -1.011842131614685, 0.035634834319353104, -1.2153680324554443, 0.46829771995544434, -0.24258357286453247, 0.5568339824676514, -0.9615959525108337, 0.8175014853477478, 1.533308744430542, 0.03425023704767227, -0.2591402232646942, -1.265802264213562, -0.4884771704673767, -0.28697508573532104, 0.28016331791877747, -1.2855727672576904, -0.31964704394340515, 0.011892491951584816, 0.3529108464717865, 0.9691683650016785, 0.7960769534111023, -1.1184954643249512, 0.03078964166343212, 0.05851215496659279, -0.1700146645307541, -0.5483618378639221, -0.12525995075702667, 0.9531245827674866, -0.15273115038871765, -0.6409571766853333, 0.021352313458919525, -2.6466283798217773, -1.5548460483551025, 0.9643106460571289, -0.26467353105545044, -0.9151992201805115, 0.39637139439582825, 2.0372214317321777, 0.2584253251552582, 2.2051515579223633, 0.34561392664909363, -0.028841007500886917, 1.7088167667388916, -2.3332343101501465, 1.0776355266571045, -1.4729866981506348, -0.490424782037735, -0.27224481105804443, -0.7171608209609985, -0.006159932818263769, 2.223783254623413, -0.2795362174510956, -1.2629233598709106, 1.0458494424819946, -0.49775680899620056, 0.14038032293319702, -0.45405396819114685, 1.4393854141235352, -2.1682889461517334, 1.9206156730651855, -0.6317608952522278, 0.25423550605773926, -0.06382838636636734, -0.9193755984306335, -0.37771520018577576, -0.8445757031440735, -0.9787368178367615, 0.5008537769317627, -1.4025286436080933, 0.3922753930091858, 0.5810364484786987, 0.8364250063896179, -1.412365198135376, 1.2307668924331665, -1.2911803722381592, 1.2095452547073364, 1.5440000295639038, -1.0909031629562378, -1.4096112251281738, 2.3962998390197754, 0.10456588119268417, 1.597245693206787, -0.27851954102516174, -0.12080416828393936, 0.16185249388217926, -0.5462283492088318, 0.5482414364814758, 0.5466734170913696, -0.10205003619194031, 0.6800324320793152, 1.4941848516464233, -1.0844000577926636, 0.04374638944864273, 0.42668288946151733, 2.395189046859741, 0.7742242217063904, 0.6437112092971802, 1.1701855659484863, -2.5730295181274414, -0.6317914724349976, -1.8828989267349243, -1.1173434257507324, 0.600856363773346, -0.8216283917427063, -0.3620631992816925, -0.7428083419799805, -0.04662257432937622, -0.7333036065101624, -1.8014302253723145, 1.2050915956497192, -1.4441585540771484, 0.15916579961776733, 0.43083932995796204, -0.2846667468547821, 1.1773606538772583, 0.4721626937389374, -0.7875639200210571, 1.4360805749893188, 1.0703541040420532, -0.5092533230781555, 0.742672860622406, -1.4728981256484985, 0.5657591223716736, -1.3144922256469727, -1.0101546049118042, 0.02017141319811344, 1.8268307447433472, -2.0008206367492676, 3.028548240661621, -0.7073136568069458, 1.891966700553894, 0.9174441695213318, 0.23276960849761963, 0.6766726970672607, -1.320111632347107, 0.37134885787963867, -0.2429344207048416, 1.0594595670700073, 0.4186939597129822, -1.9401551485061646, -0.7992469072341919, 0.8349153399467468, -2.5076229572296143, -0.2128647267818451, 0.8327039480209351, -0.09375377744436264, 0.38724276423454285, -0.7989980578422546, 0.18130670487880707, -1.2259305715560913, 0.9943097829818726, -0.43692928552627563, 0.6242756843566895, -0.6105226278305054, 0.5080750584602356, 0.8016899824142456, 1.131421685218811, 1.4942753314971924, -0.4595385491847992, -1.6304203271865845, 0.12426190823316574, -0.25046318769454956, 1.3669699430465698, 1.6157203912734985, 0.5788419246673584, -1.6654146909713745, 0.10228326171636581, 0.1911853551864624, 1.0144314765930176, -0.8683359026908875, 2.2282066345214844, -0.8996331691741943, 2.2141847610473633, -0.0663294643163681, 0.830454409122467, 0.23346929252147675, 0.5646128058433533, -0.682780921459198, 0.5794537663459778, 1.0938301086425781, 2.0613086223602295, 0.28266605734825134, -1.7789270877838135, 0.8801969885826111, 0.965510368347168, 0.934615969657898, -0.4085274338722229, 0.6900537014007568, -0.4970587491989136, -0.11418122053146362, -0.5391548871994019, 2.002174139022827, -0.28158459067344666, 0.4183979034423828, 0.935670793056488, -1.1694486141204834, -0.09864602237939835, -0.4975893497467041, 0.009825901128351688, -0.19447311758995056, -0.9949479699134827, 0.7707317471504211, -0.21306449174880981, 0.4201817214488983, -0.13045847415924072, -2.4239556789398193, 0.19943824410438538, -1.353735089302063, 0.00014585712051484734, -0.3976987898349762, -1.1211228370666504, 0.23835869133472443, 0.4098251461982727, -0.07098431140184402, 0.03439745679497719, -0.16213296353816986, -0.024650990962982178, -1.5365098714828491, -0.5395533442497253, -1.6356921195983887, 0.5648518204689026, 1.507192850112915, -0.0052831294015049934, 2.5085463523864746, -0.7097203731536865, 0.15992610156536102, -1.362910270690918, 0.08503777533769608, -1.7055388689041138, 0.1212131604552269, -1.2307162284851074, -0.7992089986801147, 1.3076146841049194, 0.8358989357948303, 0.5817789435386658, 0.21803870797157288, -0.5911847949028015, 0.9075381755828857, -0.2737358510494232, 1.3893898725509644, 0.7671011686325073, 1.6089341640472412, 2.831559419631958, -2.0089597702026367, 0.5193098187446594, 1.139968991279602, 0.7228372097015381, -0.5682756900787354, 0.7608294486999512, -0.43111175298690796, -1.3149794340133667, -0.6044138669967651, 0.16883650422096252, -0.3431425988674164, 0.6840133666992188, 0.035146769136190414, -0.1467047929763794, 0.24529406428337097, 0.2470381110906601, 0.017891619354486465, -0.4563271105289459, -1.0532792806625366, -0.34354960918426514, 1.5080628395080566, -1.5015382766723633, 0.5090601444244385, 0.48200374841690063, -1.3897864818572998, -1.3700376749038696, -0.7658504247665405, 0.621254563331604, -0.35996493697166443, -0.1664007157087326, -0.0687655434012413, 2.456024169921875, 0.4005867838859558, 0.19666454195976257, -2.216745376586914, 0.323968768119812, -1.2533007860183716, 1.6485278606414795, -1.2051427364349365, -0.6370542049407959, -0.21036915481090546, 0.966365396976471, 0.8107444047927856, 2.494232416152954, 0.5696865916252136, -0.5269327759742737, -1.253358006477356, -0.5349804759025574, 0.3685205578804016, 0.49900656938552856, 0.29646778106689453, -0.28368374705314636, 0.14414741098880768, -0.5488273501396179, 1.1037015914916992, 0.015028491616249084, -1.1968774795532227, 1.8130028247833252, 1.588659405708313, -1.9816248416900635, 0.7215626239776611, -0.5520135760307312, 0.8535329103469849, -1.0935908555984497, 0.5065135359764099, 0.39778512716293335, -1.064650297164917, -0.6854726076126099, 0.9923794269561768, -0.5070511102676392, 0.6829046010971069, 1.0105115175247192, -0.16520437598228455, 0.6667664051055908, 0.8838252425193787, -0.5048532485961914, -0.33627066016197205, -0.37023982405662537, 0.10162260383367538, -0.31127670407295227, 0.35599952936172485, -0.14906854927539825, 0.3272973299026489, 0.3734775185585022, -0.26147204637527466, -0.5537014603614807, -0.3140326142311096, -0.2048504799604416, 0.40776365995407104, 0.010865502990782261, 0.634071946144104, -1.7010642290115356, -0.05591816455125809, -1.4201914072036743, 0.6006197929382324, 0.4351285398006439, 0.10832582414150238, -0.17352943122386932, -0.04344971850514412, 0.5305041670799255, 0.12661178410053253, -0.0026651371736079454, -0.1846395879983902, -0.6123892068862915, -2.221135377883911, 0.17718079686164856, 1.8410508632659912, 0.9121562838554382, -0.35812151432037354, -1.1459113359451294, 0.1682223677635193, -1.5579993724822998, 0.2613142728805542, -0.8999795317649841, -1.2176387310028076, -0.34975627064704895, 0.6805973649024963, 0.10169468075037003, 1.2767887115478516, -0.7510563731193542, 1.1129038333892822, 1.0563843250274658, -0.49412208795547485, 0.13340990245342255, 0.12462130188941956, 1.4666212797164917, -0.8985111713409424, 1.478545069694519, -0.8033244013786316, 0.4296450614929199, 0.6420938372612, -0.6345075368881226, -0.22959968447685242, 1.2757892608642578, -0.8530169129371643, -0.5822122693061829, 2.250046491622925, 1.2727299928665161, -2.433476448059082, 0.563708484172821, 1.019295334815979, 1.0182446241378784, -0.26886874437332153, -1.0688153505325317, -1.0952820777893066, 0.3571988642215729, -1.08940589427948, -0.10110976547002792, 1.461572527885437, -0.7072445154190063, -0.365499347448349, -1.5315883159637451, 0.4243958294391632, 1.3509244918823242, 1.2004412412643433, -0.3154543936252594, -1.1048915386199951, 0.07177125662565231, 0.7968284487724304, -0.9100900888442993, -1.6920486688613892, 0.9371957778930664, 0.07129748910665512, 0.07427655905485153, 0.5761763453483582, 1.9979008436203003, -0.3117768466472626, 0.27647873759269714, 0.6799080967903137, 0.6839882731437683, -0.8519944548606873, 1.6875718832015991, -0.4512973427772522, -0.6047379374504089, 0.9725996255874634, 0.7623659372329712, 0.4973815381526947, -0.6575144529342651, -0.10095366090536118, 0.8719668984413147, 0.02017519250512123, -0.24440231919288635, -0.48868003487586975, -1.178721308708191, -0.15844646096229553, -0.7642608284950256, 1.5569003820419312, 0.5584977865219116, -1.8487564325332642, -0.051847875118255615, -0.8214825391769409, -0.7620190382003784, -0.2030401974916458, 1.4557700157165527, -0.39075592160224915, 0.43501973152160645, 0.1670922189950943, -0.5899154543876648, 1.3271219730377197, -0.2772027552127838, -2.0014593601226807, 0.5865994095802307, -0.5998307466506958, 0.0981961190700531, -0.20435817539691925, 1.4194366931915283, -1.334789752960205, -1.4620928764343262, -1.4827380180358887, 0.6814030408859253, -0.8171128630638123, -1.7336257696151733, -0.7952538132667542, 0.4698903262615204, 0.1608067899942398, -0.7726560831069946, -0.3437785804271698, -0.6032715439796448, 0.7638676762580872, -0.2727823853492737, -0.3999347388744354, 0.6934128403663635, -0.13347212970256805, 1.5490269660949707, -0.5426508188247681, -0.29642459750175476, -0.11785324662923813, 0.044080913066864014, 1.6445324420928955, -0.7226622700691223, 1.1366474628448486, -0.420625776052475, -0.962722659111023, 1.4777168035507202, -0.8852565884590149, -0.008876015432178974, 1.2426753044128418, -0.1711452752351761, 0.5758764743804932, -0.14726868271827698, 1.5479365587234497, 0.49865782260894775, -0.15887315571308136, 0.34555238485336304, 1.2998950481414795, 0.5634186267852783, -1.950635313987732, 1.8030166625976562, -0.6672829985618591, 1.2960377931594849, 0.16503843665122986, -1.2040715217590332, 1.4023274183273315, -1.7031253576278687, -0.6921641230583191, -1.2753411531448364, -2.018707752227783, 1.5029526948928833, 0.6402708888053894, 0.7338082790374756, -0.7712514996528625, -0.7522240877151489, 0.5886398553848267, -0.28994256258010864, 0.6289765238761902, -1.4917213916778564, -0.2705458700656891, 1.5127241611480713, -0.9270451664924622, -0.21556919813156128, 0.4333558678627014, 0.5676893591880798, -1.227786660194397, -1.8397846221923828, 0.5880715847015381, 0.45835766196250916, -0.13699118793010712, 0.47633427381515503, 0.5117881298065186, 0.0014326577074825764, -0.7696585655212402, -1.7032263278961182, 0.9633045196533203, -1.6084898710250854, 0.49911344051361084, 0.14821553230285645, -1.0955899953842163, -0.6297403573989868, -2.278782606124878, -1.2009669542312622, 1.7462007999420166, 0.2955222427845001, 0.9402610063552856, -0.9283685088157654, 0.8157309889793396, -1.7744868993759155, 0.929774820804596, 0.31765714287757874, -0.42250317335128784, 1.0707460641860962, 1.653890609741211, 0.3803933560848236, -0.09035221487283707, 0.4369407296180725, -0.011383741162717342, 0.2039278894662857, 1.6165711879730225, -0.21280938386917114, 0.9045305252075195, -0.21719758212566376, -1.6539093255996704, 0.7690181732177734, -2.2894253730773926, 0.33937203884124756, -0.07873721420764923, -1.1221847534179688, 0.7313024401664734, -0.22195588052272797, 0.3807224929332733, -1.704526662826538, 0.7364181876182556, 0.22496730089187622, 0.9538780450820923, -1.5062726736068726, 0.8673202395439148, 0.07823703438043594, 0.3903238773345947, -0.4251236319541931, -1.7593096494674683, -0.17672772705554962, 0.5322518944740295, -0.3642798960208893, 0.4705389738082886, 1.3894948959350586, 0.24398906528949738, 0.4536573588848114, 1.1235049962997437, -0.052221499383449554, 0.13914775848388672, -0.6326781511306763, 0.9401025176048279, -0.8482856750488281, -0.300847589969635, -1.21702241897583, -0.032464444637298584, -2.063642978668213, -0.5369800925254822, 0.8097928166389465, -0.2094438672065735, -2.2946858406066895, -0.1981194168329239, -0.3988446593284607, -0.9079197645187378, 1.8204389810562134, 2.3748409748077393, 0.4878287613391876, -1.1596635580062866, -0.1879270076751709, -1.0486552715301514, 0.15801118314266205, 0.5139785408973694, -0.5017533898353577, 1.0201435089111328, -0.16046836972236633, -1.2134947776794434, -0.13362061977386475, 0.32426679134368896, 0.6267533898353577, 1.0367166996002197, 0.8228299021720886, 0.5221970081329346, -0.8100250959396362, 0.5099757313728333, -0.1364658623933792, -1.878843069076538, -0.4991122782230377, 0.2348068505525589, -0.04002484679222107, -0.3280668556690216, 0.914742112159729, -0.6736530065536499, -0.9085277318954468, -0.0890246331691742, 1.2521198987960815, -0.33390873670578003, -0.05153710022568703, -0.7868699431419373, -0.7555893063545227, 2.2455170154571533, 0.9287542700767517, 0.004791000857949257, 0.5477112531661987, -0.8324769139289856, 1.020627498626709, 0.39530184864997864, 0.07820907980203629, -0.16056720912456512, -1.7897409200668335, -0.8486839532852173, -0.8758546113967896, 1.3406041860580444, 0.5041102766990662, 1.2012419700622559, 0.8686818480491638, 0.03686615079641342, 1.2858344316482544, 0.6801254749298096, -0.7110012173652649, 1.1128630638122559, 1.5213733911514282, -1.8601585626602173, 0.4325157403945923, -1.0313407182693481, -0.8155009746551514, -0.09670888632535934, -0.6524563431739807, -0.4850981533527374, -0.6699109673500061, -0.15064746141433716, -0.03341671824455261, -0.13357867300510406, -0.8840826749801636, 0.13579417765140533, 0.37384581565856934, -1.6177371740341187, 0.6239797472953796, -0.46189016103744507, -0.6142787337303162, -0.03640222176909447, -0.11371096223592758, -0.25446617603302, -1.1563069820404053, 0.6111907958984375, 0.1773586869239807, 0.009609567932784557, -0.3200940787792206, 2.506108283996582, 0.7692628502845764, -1.0760568380355835, 1.4021754264831543, 0.21628877520561218, -1.3624472618103027, 0.9494374394416809, -0.2500794231891632, 0.022676343098282814, 0.44839078187942505, 0.08919868618249893, 0.5452189445495605, -1.354528546333313, -0.022576753050088882, 1.9704318046569824, 0.46655037999153137, -0.4348648190498352, 1.2657616138458252, 0.6170275807380676, -1.1236387491226196, -0.2542808949947357, 1.1538997888565063, -0.2791852653026581, 0.06148729473352432, 0.40215903520584106, -0.30148035287857056, 0.1560898870229721, 0.11175991594791412, 1.2408499717712402, -0.3360637426376343, 0.31022587418556213, -0.7326685190200806, 0.5028172731399536, -0.9767342805862427, -0.5927940011024475, 1.3111118078231812, -1.221876621246338, 1.0036271810531616, 0.41260138154029846, 0.4996981620788574, 0.8462446928024292, 0.18854621052742004, -0.28139355778694153, 0.5286327004432678, -1.1682788133621216, -0.44463449716567993, -0.01479671522974968, -1.5348527431488037, -0.45080551505088806, 0.4814911484718323, -0.8317659497261047, 0.5826376676559448, -0.9458991289138794, 0.32345050573349, 0.561772882938385, -1.9026837348937988, 2.1323938369750977, -0.2767065167427063, 1.9690805673599243, 0.1007433831691742, -0.6171929240226746, -0.46167969703674316, 0.9847232103347778, 2.0278992652893066, -0.9814199209213257, 0.006943982560187578, 0.26825931668281555, -0.5770401358604431, 0.5915404558181763, 0.42279455065727234, -0.5416151881217957, 0.6040812134742737, 0.052429795265197754, 1.0906615257263184, -0.15031857788562775, 0.8682302832603455, 0.29792171716690063, 1.3892881870269775, 1.1936430931091309, -0.6918847560882568, 0.10137537121772766, 0.3938971757888794, -0.3829073905944824, 1.1649916172027588, 0.5017687082290649, -1.6640632152557373, -0.9771649837493896, 0.5214604139328003, -2.04707670211792, 0.4349062442779541, 1.9445805549621582, 2.540795087814331, -0.010433444753289223, -0.030180249363183975, -0.7620052695274353, -0.11376260966062546, 0.5009694695472717, 2.1391797065734863, 1.951261043548584, -0.43922051787376404, 0.01680048182606697, 0.5479159951210022, -1.308811068534851, -0.6811296343803406, 0.03233899548649788, 0.3976461589336395, -0.6254139542579651, -0.48806262016296387, 0.38412371277809143, 0.05480452999472618, 0.5834049582481384, -0.43569764494895935, -0.32498031854629517, 0.8588112592697144, -0.2888758182525635, 0.9637183547019958, -0.4681703746318817, 0.8538077473640442, -0.04201630875468254, -0.8653789758682251, 0.31051549315452576, 0.7113084197044373, 0.2614211440086365, 1.485126256942749, -0.630331814289093, 0.6384952664375305, 0.3427814245223999, 0.8161208033561707, -1.5477911233901978, -0.5149103403091431, 0.8674904704093933, -0.9890310764312744, -0.22497917711734772, -0.33559706807136536, 1.8367000818252563, 0.2681943476200104, 0.386381596326828, 0.8299046754837036, 0.4259835481643677, 0.016617590561509132, -0.7609184384346008, 1.0315883159637451, -1.8578978776931763, 0.4115421772003174, 0.3876853585243225, -0.5362902879714966, 0.45958173274993896, -0.7854098677635193, 0.04992952570319176, 1.3720649480819702, -0.9278167486190796, -2.0025858879089355, 1.033628225326538, -0.8469927310943604, 1.8165247440338135, -0.4618918001651764, -0.6998922228813171, -1.0736521482467651, -0.08735533803701401, -0.23432722687721252, -0.3417722284793854, 0.5492861866950989, -0.6869232654571533, 0.6221966743469238, -0.333379864692688, -1.436822772026062, -2.2595129013061523, -0.2946673035621643, -1.2279033660888672, 1.8948150873184204, -0.23617634177207947, -1.6305121183395386, 0.29812756180763245, -0.5038817524909973, -1.0534731149673462, 0.42212975025177, 0.016855601221323013, 1.0903948545455933, -0.9890692234039307, -0.1004774272441864, 0.26450851559638977, -0.9010166525840759, -1.758651614189148, -1.133103370666504, 1.8927218914031982, 0.861940860748291, -0.07142063975334167, 1.7284477949142456, -0.03676186129450798, 0.8567035794258118, -0.1394832581281662, -0.8793424367904663, 1.9145084619522095, 1.160192847251892, -0.3230292499065399, 1.5539827346801758, -1.5332930088043213, 0.08358560502529144, -0.44931602478027344, 1.5464805364608765, 1.1455148458480835, -0.12447335571050644, 1.935143232345581, -1.6200287342071533, -1.7323107719421387, -0.14238324761390686, 0.8500115275382996, 0.5495496988296509, 0.4903789460659027, -0.895467221736908, -0.1089385598897934, 0.10708251595497131, -1.668329119682312, 1.316385269165039, -1.6094114780426025, -0.5123268961906433, 0.1159229502081871, 1.4474223852157593, -1.8503000736236572, 0.04401081055402756, 1.6891852617263794, 0.49439141154289246, -0.6570839285850525, -0.023339152336120605, -1.6555249691009521, 0.2985759377479553, 0.5654192566871643, -0.8864118456840515, -1.9625808000564575, -0.8349482417106628, -0.3491913080215454, 1.8387296199798584, -0.4473780393600464, 0.007264395710080862, -0.1025693416595459, -0.5885274410247803, 0.07159239053726196, -0.08384007215499878, -0.44524121284484863, 1.138198733329773, -0.6225728392601013, -0.2656753659248352, 0.13212555646896362, -0.3464677631855011, 2.337416648864746, -0.43353143334388733, 1.0331496000289917, -1.716792106628418, -0.8027616143226624, -0.16018140316009521, 0.6023972034454346, 0.8046334385871887, -0.33311712741851807, 0.163357213139534, -0.9791330099105835, 0.6748325228691101, -0.1020570769906044, -0.0894785150885582, 0.5771381258964539, 0.28657564520835876, 1.334364891052246, -0.46470630168914795, 1.834449291229248, -0.18222905695438385, -1.0931047201156616, -0.567717432975769, 0.39204320311546326, -0.09885643422603607, -0.2644636034965515, -1.5459026098251343, 0.3781157433986664, -0.35404399037361145, 2.12795352935791, 0.8177099227905273, 1.7947118282318115, 0.053311266005039215, -0.6099158525466919, -0.15796446800231934, -1.222314715385437, 1.0969407558441162, -0.6966670155525208, -0.9385878443717957, -0.12485717982053757, 0.15718884766101837], [-1.9760912656784058, -0.9630528688430786, 2.2002744674682617, 0.8502337336540222, -1.1437667608261108, -0.26000434160232544, 0.2633111774921417, -0.13933181762695312, 0.5512582659721375, 0.573135256767273, -1.595777153968811, -1.1244014501571655, 0.328862726688385, -0.5345962047576904, -0.9022035002708435, 0.9735705256462097, 1.6774961948394775, -0.4545224606990814, -1.2205369472503662, 0.729099452495575, -0.6630317568778992, 0.6646297574043274, 1.3738791942596436, 1.5043599605560303, 0.37864166498184204, 1.3852784633636475, 1.1881542205810547, 0.10323671996593475, 0.34409838914871216, -0.14907000958919525, -0.13931824266910553, 2.0299129486083984, -0.18585889041423798, -1.596670389175415, 1.4385106563568115, -1.2581990957260132, -1.6902891397476196, 0.4308915138244629, -1.454304814338684, -0.3991473615169525, 1.0036121606826782, 1.526228427886963, 1.3651520013809204, -0.3224218785762787, -1.0289949178695679, 1.0769927501678467, 0.06022903323173523, -2.5477538108825684, -0.31726980209350586, -0.43552926182746887, -0.3252451419830322, -0.8670948147773743, 0.10322012007236481, 0.06711907684803009, -1.0229655504226685, 1.7767757177352905, 0.9052075743675232, 1.0241520404815674, -1.6507823467254639, -0.5796510577201843, 0.935956597328186, -0.4656899571418762, 1.42231023311615, -0.9999262094497681, 1.4007954597473145, -1.4831737279891968, 0.23970827460289001, 1.6243313550949097, -0.02864229306578636, 0.0886070504784584, 0.14825153350830078, -0.8019494414329529, -0.48536789417266846, 0.5597299337387085, 1.0883777141571045, -1.4997273683547974, 0.47469425201416016, 1.081697940826416, -0.28979596495628357, 1.8512327671051025, -2.799901008605957, -1.92489755153656, -0.44547009468078613, 0.5846943259239197, 0.21757011115550995, -0.0440603643655777, 2.23647141456604, 0.8300812244415283, -1.593069314956665, -1.2304167747497559, 0.8224231600761414, -0.2817457616329193, -0.5938539505004883, 1.0378504991531372, -0.4376557767391205, -1.0448182821273804, 1.5411596298217773, 0.5004610419273376, -0.6982054710388184, 1.0263845920562744, 0.3099769949913025, -0.1917075514793396, -1.3392304182052612, -0.8195451498031616, -0.26276594400405884, 0.8849186301231384, -0.4937964379787445, -0.8272625207901001, -0.23453904688358307, -0.14275401830673218, -0.7541828155517578, 0.07503924518823624, -1.605653166770935, 1.6569799184799194, 0.30703550577163696, 1.9118543863296509, -0.10435398668050766, 0.6153364777565002, 0.13271808624267578, -1.4772706031799316, -0.34566742181777954, -1.1757477521896362, 0.37402981519699097, -0.7837867140769958, 1.1078511476516724, 0.3719766438007355, -1.508453130722046, -1.4049943685531616, 1.3714321851730347, 1.6931487321853638, -1.3617804050445557, -1.951981782913208, -1.6074570417404175, -1.1252256631851196, 1.7703897953033447, 1.5323113203048706, -0.06515926122665405, 0.10590995848178864, -1.0182099342346191, -0.6146262884140015, -0.1883223056793213, 0.657050609588623, -0.44010886549949646, -1.0526295900344849, 1.964374303817749, 0.9033142924308777, 2.5397720336914062, 1.488253116607666, 0.050045158714056015, -0.13290652632713318, -0.40722450613975525, -1.4128268957138062, -1.555928349494934, 1.1696256399154663, 0.13706713914871216, -0.10129281133413315, -0.834182858467102, -0.5680890083312988, -0.8177060484886169, -0.3498012125492096, 0.30821844935417175, 0.21184439957141876, 1.4922499656677246, 0.4384880065917969, 0.9371993541717529, -0.6830233335494995, -1.023780107498169, -0.7198159694671631, 1.3954010009765625, -0.557614803314209, 0.9179584383964539, 0.5721120238304138, 0.4281614422798157, 0.014942863956093788, -0.26862892508506775, 0.021538063883781433, -0.9050663709640503, 0.4173147678375244, 1.7733983993530273, 1.2941967248916626, -0.24777065217494965, 0.758265495300293, 0.40696340799331665, 0.27753448486328125, -0.5885351896286011, -0.2149551659822464, 0.4949093759059906, -1.3783552646636963, 0.16536347568035126, 0.11134522408246994, 1.0510565042495728, -0.7185758352279663, -0.5488165020942688, 0.022232821211218834, 0.4161537289619446, 0.27090227603912354, 0.3703259825706482, -0.5517779588699341, 1.0510808229446411, 0.03869899362325668, -0.7714786529541016, -0.08207637071609497, -0.3663005828857422, 1.541967749595642, 2.5631256103515625, -0.13434860110282898, -0.9519010186195374, -3.071082830429077, 1.0022324323654175, -0.8158431053161621, 0.5147892832756042, -0.2776601016521454, 0.40929287672042847, 0.841001033782959, 0.8189435005187988, 0.061340346932411194, 0.6738750338554382, -0.2936045825481415, 0.02118387445807457, -0.7301111817359924, 0.470663458108902, 0.03891867771744728, -0.7963840961456299, -0.725679874420166, 0.7277053594589233, -1.3524339199066162, -0.48819059133529663, 1.0811700820922852, 0.32923179864883423, 0.2731301784515381, -0.2849031090736389, 0.9163026213645935, -1.1304908990859985, 1.1312077045440674, 0.49241459369659424, -0.3081880807876587, -1.3884832859039307, -1.1679171323776245, 0.13938716053962708, 1.4635636806488037, -0.008589532226324081, 1.4678807258605957, -1.4882869720458984, -0.32979273796081543, -0.5051062107086182, 0.03143012896180153, 1.0889086723327637, -0.87645024061203, 0.15721799433231354, -0.7112065553665161, -0.4382384121417999, -1.5127304792404175, -0.23075519502162933, -1.07943594455719, 0.9905524253845215, 0.057536911219358444, -0.39507216215133667, 1.6579322814941406, 1.4126816987991333, 1.0761971473693848, 0.09777950495481491, 1.3572129011154175, -0.41820472478866577, 0.9518725872039795, 0.055893734097480774, 1.5685430765151978, -1.4922388792037964, -0.9022153615951538, -1.5407661199569702, 0.4357921779155731, 0.7109988331794739, -0.401872456073761, 0.23569926619529724, -0.18824264407157898, 0.49699866771698, 0.4925549328327179, -1.052366018295288, -1.3848820924758911, -1.1252684593200684, 0.5445351004600525, -0.9540776014328003, -0.946884036064148, 0.7244316935539246, 0.8645408749580383, -1.6701924800872803, 2.1649997234344482, 1.3191972970962524, -0.2004014551639557, -0.38521257042884827, -0.22576138377189636, -1.2680959701538086, 0.23541873693466187, 0.6891963481903076, 1.6764488220214844, 0.40443357825279236, -0.3966556489467621, 1.602383017539978, 0.58943110704422, -0.060287632048130035, 2.12469482421875, 0.35844939947128296, 0.5897210240364075, -0.11232984811067581, 0.2527974247932434, 0.6971657276153564, 1.3478362560272217, 0.2956557273864746, -0.40048059821128845, 1.3527066707611084, -0.11338445544242859, -0.3880572021007538, -1.631409764289856, 0.040563762187957764, 0.8292953968048096, 0.797275185585022, 1.3324261903762817, -0.33722439408302307, 1.6798691749572754, -0.4736742377281189, 1.4369897842407227, 1.1031159162521362, 1.0136868953704834, 1.191495418548584, 0.7272389531135559, -0.5596644282341003, -0.2572696805000305, 1.9463001489639282, 0.793266773223877, 1.324757695198059, 1.3433449268341064, -0.885801374912262, 1.0432642698287964, 1.137414813041687, -0.8249873518943787, -1.5479947328567505, 0.8917495012283325, -1.105114221572876, -0.6790180802345276, 1.2609343528747559, 0.9338917136192322, -0.47701162099838257, -0.6746624708175659, -0.11064794659614563, 0.47249898314476013, -0.4898454248905182, 1.2745310068130493, 1.8159456253051758, -0.823101282119751, 0.20663215219974518, -0.25498244166374207, 0.2793225347995758, 0.04947599023580551, 0.9081965684890747, 0.19004030525684357, 0.10527680814266205, -0.8668391704559326, -1.0255019664764404, -0.5383613109588623, -0.23071423172950745, -0.7042674422264099, 0.9282070994377136, -1.1037263870239258, 1.2709083557128906, -1.208877444267273, 1.0549225807189941, -0.44498851895332336, -1.9724904298782349, -0.22523756325244904, -0.3563167154788971, 0.429217666387558, 0.8587892651557922, -1.674082636833191, -0.1539880484342575, 0.3002452850341797, 0.37484630942344666, -0.8308115601539612, 0.1746944785118103, 0.35102683305740356, 0.16919398307800293, -0.2962574064731598, 0.2957427501678467, -0.11651153117418289, 0.4941968321800232, -0.3259153664112091, 1.7167046070098877, -0.21116118133068085, -0.4547688066959381, 0.44448497891426086, -0.32543158531188965, -0.09479250013828278, -0.8072980642318726, 0.4788762927055359, -1.0584139823913574, 0.7726258635520935, -1.3313424587249756, -0.32495367527008057, 0.5668713450431824, -0.46035632491111755, 0.23567445576190948, 0.9073130488395691, -0.24772228300571442, -1.0460866689682007, -0.315925657749176, -0.5328159332275391, -0.49195998907089233, 0.3188404142856598, 0.12013760209083557, -0.6435576677322388, -1.36082124710083, 0.9976356029510498, 0.8358471989631653, 1.6575727462768555, -0.0806211307644844, 0.4326290488243103, -0.564793050289154, 0.2999107539653778, -0.03380797430872917, 0.013104322366416454, -0.041699111461639404, 0.018280059099197388, 1.0655790567398071, 0.7983068227767944, 0.4592773914337158, -0.0798996090888977, -0.3696255683898926, 0.1469418853521347, -1.0568047761917114, 0.19427195191383362, -0.6629663705825806, -0.8937861919403076, -1.0222949981689453, -0.2890787124633789, 1.219925880432129, 1.4634021520614624, -0.8189215064048767, -0.10678082704544067, 0.30829283595085144, 0.0433291457593441, -1.0766732692718506, 0.5275799036026001, -1.5618997812271118, -0.5555519461631775, 0.19772695004940033, -0.47184818983078003, 0.019933490082621574, 1.0652518272399902, 0.20832988619804382, -0.5661653876304626, 1.2528738975524902, -1.2875463962554932, 1.000357985496521, -1.2064158916473389, -0.008169615641236305, -1.6347894668579102, -0.39354392886161804, 1.11137855052948, -0.03373143449425697, -2.4866766929626465, -1.7217686176300049, -0.40353482961654663, -0.1358754187822342, -0.06851949542760849, -1.3148353099822998, -0.027094872668385506, -0.09458460658788681, 0.20641909539699554, -0.43501579761505127, 0.553684651851654, -1.0212388038635254, 1.5383158922195435, -1.0231575965881348, -0.17160172760486603, 0.38853612542152405, -1.9055793285369873, 1.114701271057129, -1.6824963092803955, -0.5902922749519348, 2.2274065017700195, -0.04562851041555405, 0.40894898772239685, -2.0854368209838867, 2.2589261531829834, -0.9202232360839844, -1.3167232275009155, -0.12213411927223206, -0.6516655683517456, 0.780654788017273, -0.8996286988258362, 0.788286030292511, -1.2061558961868286, -0.3671606481075287, 0.6581982970237732, 0.5532899498939514, 1.7265230417251587, 0.2102309912443161, -1.178446650505066, -1.7482757568359375, -0.6008734107017517, -0.3980092406272888, -0.9535598158836365, -2.0045113563537598, 0.11309252679347992, -0.6550067067146301, -0.10438244789838791, -0.8174623250961304, 1.7880959510803223, -0.3891042470932007, -1.9139631986618042, -0.14577138423919678, -0.7201419472694397, -0.1962781697511673, 1.0157893896102905, 0.14992691576480865, 0.16492959856987, -0.24583487212657928, -0.0791405737400055, 0.00837119109928608, 0.06600024551153183, 0.5565372705459595, 1.0225507020950317, 2.2467172145843506, 0.3569544851779938, 1.6641470193862915, -1.1949561834335327, -0.4124148488044739, 0.09290120005607605, -0.7405731678009033, -0.3953659236431122, 0.7377054691314697, 1.1674418449401855, 2.6041111946105957, -0.2138441801071167, -0.6167288422584534, -0.021981438621878624, 0.8748739361763, -1.153696894645691, 1.2551277875900269, 0.4440821409225464, 1.4448015689849854, 0.11684141308069229, 0.47549277544021606, -1.2124924659729004, -1.03687584400177, 0.9962001442909241, -1.5505471229553223, 0.7926670908927917, -1.8024632930755615, -0.09035824984312057, -0.20346401631832123, -1.8020918369293213, 0.9367339611053467, 0.21687205135822296, 0.34758925437927246, -0.06435652077198029, -0.5760317444801331, 0.7836169004440308, -2.22892165184021, -0.7036713361740112, -0.33742862939834595, 1.0783324241638184, 1.3216333389282227, 1.0774059295654297, 0.4587344825267792, 1.7598997354507446, -1.6015089750289917, -0.16946421563625336, 0.789059042930603, 0.9574231505393982, -0.7066814303398132, 0.6867952942848206, -1.0572212934494019, 0.32909658551216125, 1.2412604093551636, 1.4059193134307861, -0.552196741104126, 0.22769318521022797, 0.14131800830364227, 0.2826908230781555, -1.2655774354934692, 0.8303706645965576, -0.21338286995887756, 0.8440632224082947, 0.8525063991546631, 0.31840094923973083, 0.1103978380560875, 0.16545552015304565, -1.2357573509216309, -0.8984419703483582, -0.5049698352813721, -0.7273541688919067, 2.7680211067199707, 1.7157515287399292, -0.26119959354400635, 0.4874283969402313, 0.3195030391216278, 1.427477240562439, 0.3761005103588104, -2.049450635910034, -0.5864014625549316, 0.38639000058174133, -1.6386029720306396, -0.5765045881271362, -0.5621969699859619, 3.026412010192871, -0.2344900518655777, -0.21140150725841522, -1.4031168222427368, 0.47040101885795593, -0.8122363686561584, -0.2084701657295227, -1.1750257015228271, -0.7732083797454834, -0.8837481141090393, -0.8787307739257812, -0.850092887878418, 0.08691874146461487, 1.3015761375427246, 2.133622646331787, 0.8330968022346497, -0.8164762854576111, -2.077112913131714, -0.31400662660598755, 0.4179519712924957, 0.19846777617931366, -0.49036577343940735, 0.09586114436388016, -0.2720584273338318, -0.319412499666214, -1.6605030298233032, 1.4706369638442993, 2.109861135482788, 0.11572103202342987, -1.615766167640686, 0.7810977697372437, -1.3174511194229126, -1.8118317127227783, 0.08644666522741318, -0.7003445029258728, 0.8825779557228088, -0.8129314184188843, -0.8405690789222717, 1.180946946144104, 0.18196028470993042, 1.0224385261535645, 0.3104534447193146, 0.34683701395988464, 0.6044185757637024, -1.2559142112731934, -1.7220959663391113, -2.5802724361419678, 0.025917287915945053, -0.1980418562889099, 0.247669979929924, 1.1240288019180298, -1.0598974227905273, -0.035652488470077515, 0.2375599592924118, 1.0456538200378418, -1.150148630142212, 0.07379189133644104, 0.13568434119224548, 1.8135179281234741, -1.0303139686584473, 0.9632543921470642, -0.7923382520675659, -0.19035334885120392, 0.9974094033241272, -0.7719930410385132, 2.1150989532470703, -1.051579475402832, -0.05300059914588928, -0.7158324122428894, 0.09375126659870148, 1.142142415046692, -0.913530170917511, 0.0314435176551342, 0.4299367666244507, 0.4612838327884674, 0.3859032988548279, 0.16728456318378448, 0.6088756918907166, -1.8012280464172363, 1.5226085186004639, 2.1049582958221436, 1.5442174673080444, 1.8773276805877686, -0.5787591934204102, -0.3867443799972534, 1.740896463394165, 1.6483159065246582, 0.5542156100273132, -0.9938843250274658, -0.5962386131286621, -0.6582967638969421, 0.7992169260978699, -0.6646146774291992, 1.3944138288497925, 1.2101807594299316, -0.2731853425502777, -1.0635793209075928, -0.08537526428699493, 0.09148135781288147, -0.013365077786147594, 0.5114320516586304, 0.770382285118103, 0.09007277339696884, -0.1207650899887085, 0.5840413570404053, 0.7249523997306824, -0.8509376645088196, 0.6975891590118408, 0.774931013584137, -0.4415808916091919, -1.1652415990829468, -1.2129024267196655, 2.085174560546875, -0.11434214562177658, -0.5414719581604004, 0.7959036231040955, 0.3031388223171234, -0.9247778654098511, -1.8939656019210815, -1.3722294569015503, -1.2489168643951416, 0.1930243968963623, -0.7452287077903748, -0.5576311945915222, 3.264798402786255, 0.752698540687561, -0.9075899720191956, 0.44565096497535706, 2.681795597076416, -1.0214327573776245, -0.6081613898277283, 0.5160515904426575, -0.005982194095849991, -0.6436246633529663, 0.7357870936393738, 0.37918010354042053, -0.460869163274765, 0.038969311863183975, -0.4335748553276062, -0.8170811533927917, -0.33534178137779236, -0.4753343462944031, 0.6100977659225464, -0.6223306655883789, -0.10065393894910812, 0.6208342909812927, 0.5251594185829163, -0.7823572158813477, 1.4038541316986084, -0.9010441303253174, -0.7388731241226196, -0.9031124711036682, -1.3310368061065674, 0.5811643004417419, 0.3332638740539551, -1.1810492277145386, -1.4833414554595947, 0.8011721968650818, -1.2085152864456177, -0.9731112122535706, -1.4138710498809814, -0.036160532385110855, -0.40131092071533203, 1.7570698261260986, 0.5709784030914307, 0.28335827589035034, -1.458717703819275, 1.2168593406677246, -1.005210280418396, 0.4188542068004608, 0.7104552388191223, 2.2629292011260986, 0.2323915660381317, 0.25168725848197937, 1.3708841800689697, 0.34172841906547546, 0.19308921694755554, 2.0704920291900635, -1.2281739711761475, 0.7313475608825684, -0.8517037630081177, -0.8621334433555603, 1.1958080530166626, 0.3091523051261902, 0.971498966217041, 1.270770788192749, -0.7710424065589905, -1.1692495346069336, 1.060363531112671, 0.5647597312927246, 0.031093604862689972, 0.7864447832107544, 1.3718132972717285, -1.23172926902771, 1.7282965183258057, -2.547146797180176, -0.8683435916900635, -0.7941462993621826, 0.3678712248802185, 0.4623187780380249, -0.04806281626224518, -0.1925785392522812, -0.32320505380630493, 0.6424256563186646, -0.2104351669549942, -0.051048725843429565, -0.9177101850509644, -0.10198406875133514, 0.5599833130836487, 1.1491215229034424, -0.024623829871416092, -0.6152738928794861, -0.4727764427661896, 1.8257193565368652, 0.8806630969047546, -0.3185233771800995, -0.13829800486564636, -0.7740764617919922, -2.688985824584961, -0.17060360312461853, -0.6434484720230103, 1.720542311668396, -0.5763028264045715, 0.1548522710800171, -0.20365342497825623, 0.5197950601577759, 0.60438472032547, 0.5618904829025269, 1.2038209438323975, 0.6147264242172241, 0.4887867569923401, -1.1614586114883423, 0.3636549115180969, 0.48356783390045166, -0.3979138135910034, 0.07992011308670044, -0.8181567192077637, -0.6998285055160522, -0.8054407835006714, -0.703509509563446, 0.11039766669273376, -0.26092374324798584, 1.1549179553985596, -0.25729265809059143, 0.3465348780155182, -0.5987745523452759, 1.3596174716949463, 0.2185918688774109, -1.3191542625427246, -0.1916736215353012, -1.1352603435516357, 1.9531614780426025, -1.4579538106918335, 0.9115172624588013, 1.1050260066986084, 0.7953075170516968, -0.14673921465873718, -0.24482007324695587, -0.9814253449440002, 1.038879156112671, -0.19330281019210815, -0.014254185371100903, 1.09841787815094, -0.7659324407577515, -0.7201419472694397, -0.22227846086025238, -0.26644062995910645, -1.074605107307434, -0.10536106675863266, 1.089333415031433, 0.7458447217941284, -0.9764774441719055, -0.7446811199188232, 1.3360941410064697, -0.316225528717041, 0.7153067588806152, -0.23328593373298645, -0.1443408578634262, -1.753278136253357, 0.5501866936683655, 0.3315957486629486, -1.2354074716567993, -0.2900010347366333, -0.11068274825811386, 0.6818364858627319, 0.8039979338645935, -0.6751704812049866, -0.9196338057518005, -0.6391648054122925, 0.3224509358406067, 0.2733234465122223, 0.8148250579833984, 1.8724427223205566, -0.09610825777053833, -2.478238582611084, 1.0083979368209839, -1.3669975996017456, 1.1484630107879639, -0.01969665288925171, 1.1918426752090454, 0.3869536221027374, -0.18721257150173187, -0.18066196143627167, 1.7632285356521606, -0.348274827003479, 0.526557207107544, -0.03579013794660568, 0.21965061128139496, 1.2236359119415283, 0.03694204241037369, -0.33667123317718506, 1.6115509271621704, 1.8704804182052612, -0.691841185092926, 2.0323028564453125, 0.39900586009025574, 0.3956187069416046, 1.2708884477615356, -0.6199082136154175, -0.11771681159734726, 1.1391690969467163, 0.03237307444214821, 0.24169643223285675, 0.8220648765563965, 0.8615484237670898, 1.2557218074798584, 0.4899531304836273, 0.974177896976471, 1.616135835647583, 2.00860333442688, 0.11783687025308609, 1.102609395980835, 0.32433846592903137, 0.05861852690577507, 1.6640654802322388, 0.18901526927947998, -0.48118701577186584, -0.9030897617340088, -0.019727889448404312, 0.6137335300445557, -1.9324363470077515, -0.7496734261512756, 0.32567763328552246, 0.008982377126812935, -1.1473764181137085, -1.2159337997436523, 0.05922190845012665, 0.05384228378534317, -2.279470443725586, 0.17715898156166077, -0.6405728459358215, 0.5023410320281982, -0.21053527295589447, 0.7437602877616882, 0.34086164832115173, 1.500707983970642, 0.5079440474510193, 0.5846537351608276, 1.2473907470703125, 0.5890923142433167, 0.26519522070884705, 0.8291903138160706, 0.7939661145210266, 0.868975043296814, -0.6985365748405457, 1.0148732662200928, 1.3509587049484253, 1.8782579898834229, 0.42383262515068054, 1.8062870502471924, -0.8399165868759155, -0.9843004941940308, 0.37110477685928345, 0.4856443405151367, 0.5696657299995422, 0.5935116410255432, 1.9087046384811401, -1.9141343832015991, -0.6480315923690796, -0.9122702479362488, -0.09338781237602234, 0.4916876554489136, 0.2784537076950073, -0.7026020884513855, -1.1859087944030762, 0.07409504055976868, 0.9565754532814026, 2.309377431869507, -0.9817907214164734, 0.8424862027168274, 0.2329181432723999, -0.429991215467453, -0.3402478098869324, 0.3753794729709625, -0.5245360732078552, 0.2639455199241638, -0.4316982328891754, -1.825750708580017, -2.516815423965454, -1.3049938678741455, 1.7213852405548096, -1.0373700857162476, -0.7843677997589111, -2.516493320465088, -0.01402734313160181, -0.5642521977424622, 0.15997453033924103, 1.0401418209075928, -1.8998526334762573, -0.1988275647163391, -0.9112730026245117, 0.6969432234764099, 0.35588183999061584, 1.3378751277923584, 2.2662148475646973, -0.529327392578125, -0.11092883348464966, -0.20248499512672424, -0.7271429300308228, -0.45030975341796875, -0.03155195713043213], [-1.4867409467697144, 1.5324506759643555, 1.3942829370498657, 0.1342548280954361, 1.6939324140548706, -0.6079220771789551, -0.14885376393795013, -0.2739686965942383, -1.499239206314087, 0.8122737407684326, -1.116784930229187, -1.5290632247924805, -1.5847207307815552, -0.9385866522789001, 0.8696362972259521, 0.6037213802337646, 0.16252557933330536, -0.335235059261322, 0.047718822956085205, 0.3386455476284027, 1.4252771139144897, -1.577963948249817, 0.9620894193649292, -0.8051356673240662, -1.5052601099014282, 0.08789574354887009, -0.7891544103622437, -1.0831681489944458, 1.179675817489624, -1.4972909688949585, -0.20078593492507935, 0.13060647249221802, 0.20892947912216187, 0.1688268780708313, -0.9302921295166016, 0.37871053814888, 0.25454390048980713, -2.5444083213806152, 2.627523422241211, 0.9279977083206177, 0.07439275830984116, -0.8565417528152466, 1.3581254482269287, -1.3340622186660767, 0.8185061812400818, -0.8140825033187866, 0.5746265649795532, -1.001707911491394, 0.6449445486068726, -0.1336900144815445, -0.5226238965988159, 0.9724154472351074, 0.10495129227638245, -0.16181017458438873, 1.8768560886383057, -0.5078176259994507, -0.9239711761474609, -0.275827556848526, 1.2577354907989502, -1.039817452430725, 0.4626200795173645, -0.7537652254104614, 0.43297192454338074, 0.845632791519165, -0.31591150164604187, 3.024294853210449, -0.5443177223205566, 0.5297198295593262, 0.9042050838470459, -0.8391846418380737, -1.183322548866272, 0.412534236907959, -0.20302538573741913, 0.7262787818908691, -0.23680998384952545, -0.08609186857938766, 0.7050642371177673, 0.4342063069343567, 0.10243073105812073, 0.6583985090255737, -1.5179733037948608, 2.249302625656128, 1.0533400774002075, 1.691237211227417, 1.0610491037368774, -0.8826894760131836, 1.6734240055084229, -0.3327859938144684, 0.7147717475891113, 0.5642666816711426, -0.3187693953514099, -0.3611281216144562, 0.9518123269081116, 0.9396144151687622, -0.4924153983592987, 0.15106414258480072, -0.8544853329658508, 0.14726534485816956, 0.9409774541854858, 1.061305046081543, 1.204656958580017, -1.083945631980896, -0.2928987443447113, 0.5882788300514221, 0.21197718381881714, 0.4047086238861084, -1.2980252504348755, -1.5147377252578735, 0.5096510052680969, -0.1811615377664566, 0.5546780824661255, -0.14861297607421875, 0.6219887733459473, -0.89520263671875, -0.15904167294502258, -0.7480539083480835, 0.0058814710937440395, -0.22526611387729645, 1.3808661699295044, -0.44553449749946594, -0.2923673689365387, 1.5123145580291748, 0.7731510400772095, 0.6059057116508484, 0.08130817860364914, 0.32743698358535767, 0.7628328204154968, -0.9384511709213257, 0.6755940318107605, 1.4448219537734985, -0.6780887842178345, 0.7316304445266724, -1.0953704118728638, -1.5939875841140747, -2.1119070053100586, -2.764962673187256, 1.8709495067596436, 0.5555143356323242, -0.33547261357307434, 0.5381758213043213, -0.10477523505687714, 0.1568257361650467, -0.28170403838157654, 2.057539463043213, 0.5769171118736267, 0.01729639060795307, 0.23064246773719788, 1.4776710271835327, -0.509380578994751, -0.24939367175102234, -0.23860077559947968, -1.7203614711761475, 0.09131373465061188, 0.935217559337616, -1.8079012632369995, -0.4896807372570038, 0.9195756316184998, 0.6785961985588074, -1.1749964952468872, -0.3308458626270294, 0.7253559827804565, -0.6046794652938843, -0.13751177489757538, 0.7629057168960571, 1.5738943815231323, 0.06579989194869995, 0.20110051333904266, 0.4650319516658783, 0.1284693330526352, 1.3413699865341187, 0.13561908900737762, -0.13195303082466125, 1.038221836090088, 0.4469231069087982, -0.7283802032470703, 1.1611273288726807, 0.8972715139389038, 0.462175577878952, -0.3621187210083008, -0.49491238594055176, 0.35349342226982117, 0.8920596837997437, 0.7011048197746277, -3.0666236877441406, -1.0338530540466309, -0.03692985698580742, -1.3633172512054443, 0.3907936215400696, 0.14147773385047913, -0.0306609645485878, 0.06318001449108124, -0.9998119473457336, -1.256215214729309, 2.197136878967285, -0.05595088377594948, 0.7746384143829346, 1.2198289632797241, -1.9951807260513306, 0.45283254981040955, -0.3037547767162323, 0.9755432605743408, 0.6696687340736389, -1.632912516593933, -1.0269380807876587, 0.7441794872283936, -0.6776601672172546, -0.24162307381629944, 0.07950016111135483, -1.2646039724349976, -1.737184762954712, 2.0643529891967773, 0.34766969084739685, -0.23138554394245148, 0.78190678358078, -1.2295736074447632, 0.6189562678337097, -0.5515621304512024, 0.06904233247041702, -1.2900071144104004, 0.3786586821079254, 0.85235995054245, 1.1837737560272217, 1.1454188823699951, 1.888505220413208, -0.8834492564201355, -0.9836083054542542, -1.8507659435272217, -0.26968270540237427, -0.4012196660041809, -0.31256183981895447, -1.9023332595825195, -0.8782870769500732, 0.8920446634292603, -0.0014078003587201238, -0.04914518818259239, -0.6093233227729797, -1.7650234699249268, -1.3868205547332764, -1.808271884918213, 0.27275460958480835, -0.5687189698219299, 1.9369181394577026, -0.13508588075637817, -0.5126069188117981, 0.49053144454956055, -1.9440172910690308, -0.2087600976228714, -0.24881400167942047, 0.09308202564716339, -0.4648030698299408, 0.4138330817222595, 1.757710337638855, -0.5712587237358093, -1.4455426931381226, -0.12415961176156998, 0.435306578874588, -0.09395445138216019, -0.9679649472236633, 1.484175205230713, 1.0170836448669434, -0.2596616744995117, 0.914219856262207, 0.11132942140102386, 0.055884744971990585, 1.6666886806488037, 2.515227794647217, 1.6644431352615356, 0.34304916858673096, 0.7761419415473938, -1.362419843673706, 0.15294837951660156, 0.9638335704803467, -0.21563288569450378, -0.495274156332016, -0.5690922737121582, 0.20494209229946136, -1.1602169275283813, -0.7821386456489563, -0.2902754843235016, 0.23043224215507507, 0.16522109508514404, 0.043223436921834946, 1.125723958015442, 0.3117591440677643, 0.001429892610758543, -0.3708346486091614, 2.014113426208496, 0.017579317092895508, -2.0487794876098633, 1.3490861654281616, 0.9255861043930054, 0.10357111692428589, -1.5286288261413574, -0.04854007810354233, -0.7829434275627136, 0.0824388638138771, 0.026292895898222923, -1.3021529912948608, 0.062142256647348404, -0.8861798644065857, 0.45443376898765564, -0.065850630402565, -1.362373948097229, -2.422116279602051, 0.45991209149360657, -1.1711887121200562, 0.8687378168106079, -0.5365515947341919, 0.6081641316413879, 1.7086299657821655, 0.4789137840270996, -0.28307169675827026, 0.9316893219947815, -2.074099063873291, -0.6450167298316956, 0.2634226381778717, 0.039085566997528076, -0.19886207580566406, 0.8288120031356812, 0.04644666239619255, 0.36892715096473694, 0.7716390490531921, -0.29928842186927795, -0.49146464467048645, 0.8480483889579773, 1.6413530111312866, 0.043905049562454224, -1.4761922359466553, 0.8408142924308777, -1.3273476362228394, 1.430630087852478, 1.853265643119812, 0.6659446358680725, 0.33958858251571655, -0.7432438731193542, -0.7188110947608948, -0.08892717212438583, -0.05679219216108322, 1.6582835912704468, -0.13140368461608887, 0.2626471519470215, 0.7989251017570496, 0.5786787271499634, -2.0506091117858887, -0.21928352117538452, 1.5979691743850708, -1.5931811332702637, 2.0217537879943848, 1.852169156074524, 1.7665231227874756, 1.3779350519180298, -0.43944087624549866, 1.214005947113037, 0.5897750854492188, 1.3905340433120728, -0.05271091312170029, 1.8777422904968262, 0.13578103482723236, -0.907055139541626, -0.4347076117992401, 0.030910689383745193, -0.5729957818984985, -0.5709505081176758, -0.3137204647064209, -0.03140023723244667, -0.17154951393604279, 1.196646809577942, -0.5969544649124146, 0.6518234014511108, -0.6832453012466431, 1.3962267637252808, 1.3878283500671387, 1.0707728862762451, 0.2850465774536133, -0.0011231419630348682, -0.24825765192508698, -1.105136513710022, 0.4576159417629242, -0.6142309904098511, 0.07761114835739136, -0.5640060305595398, -0.7167407274246216, -1.3380780220031738, 0.45363613963127136, 2.049307346343994, 1.1139473915100098, -0.06538445502519608, -0.5403849482536316, 0.4544465243816376, -0.9394045472145081, 1.0959473848342896, -0.3683875799179077, -0.7847024202346802, -0.5392001867294312, -0.968268871307373, -1.3989726305007935, -1.4086475372314453, -0.28260788321495056, 1.5480166673660278, 0.3610287308692932, -1.1269961595535278, 1.2369842529296875, -0.9566786885261536, 0.1608150452375412, 0.028779268264770508, 0.014166826382279396, -0.12642285227775574, -0.46030476689338684, -0.41227230429649353, 0.8016889095306396, 0.783115029335022, -0.5639789700508118, 1.3142037391662598, -1.7361809015274048, 0.4158090353012085, 0.834520161151886, 1.0658724308013916, -0.14422036707401276, -0.25935500860214233, -0.6002978086471558, 0.685336172580719, -0.2938251197338104, 1.891058087348938, -0.44417211413383484, 1.2307225465774536, 0.20606671273708344, -0.2928052544593811, -0.5179408192634583, 0.33381038904190063, 0.4175359904766083, 1.7763797044754028, 0.1374564915895462, 0.5542839765548706, 0.1721714735031128, 0.3179883658885956, -0.7638353109359741, -1.5745502710342407, 1.0980719327926636, 0.8648176193237305, -1.0785634517669678, 1.6039952039718628, 1.2337888479232788, 0.2626425623893738, -0.882780909538269, 0.6616517901420593, 1.1862900257110596, 0.5313055515289307, -0.9352492094039917, 1.0325204133987427, -1.585397481918335, 2.236348867416382, -0.02728888764977455, -2.19885516166687, 0.1687808632850647, 0.2644122838973999, 0.10629831999540329, 0.48115214705467224, -0.26919031143188477, -1.5318037271499634, 1.2517772912979126, 0.763094961643219, -0.10849308967590332, 0.23220007121562958, -1.0167858600616455, 1.0594490766525269, 0.37219107151031494, -0.8759709000587463, -0.028727658092975616, -1.2717089653015137, 0.38285666704177856, -0.8569769263267517, 1.4995522499084473, 0.19325387477874756, 1.79110848903656, 1.6200697422027588, 2.516282796859741, -0.9812308549880981, 0.6641514301300049, -0.8884356021881104, 1.0408239364624023, 0.3787412643432617, -1.1732407808303833, 0.02974109910428524, 0.16916169226169586, 1.408921718597412, 0.445759117603302, -0.46188950538635254, -0.1677364557981491, 1.1316113471984863, -1.3964017629623413, -0.6374375820159912, 0.32634106278419495, -0.595142662525177, -0.014901983551681042, 0.7960516810417175, 0.037630464881658554, -0.4515487253665924, 0.8952026963233948, 1.1257686614990234, -1.557765245437622, 0.035556547343730927, -1.1798815727233887, 1.4417884349822998, -0.032336074858903885, -0.6999555826187134, 1.836971640586853, 0.2075575292110443, 0.1506316214799881, 0.1549260914325714, 0.9803227782249451, 0.852116048336029, -0.5715793371200562, -0.5073887705802917, 0.8930527567863464, -0.9735035300254822, -0.4573447108268738, 0.3432597219944, -0.36731061339378357, -0.9908629655838013, 0.5767207145690918, -1.272802710533142, 0.7729538679122925, 0.5815554261207581, 0.7758530974388123, 0.4662722647190094, 1.2013659477233887, -0.7169605493545532, -0.7559446096420288, -0.12664756178855896, -1.729996681213379, 1.2310491800308228, -1.434453010559082, 0.5350724458694458, 0.4747227132320404, 0.43877917528152466, 1.199211835861206, 1.4871443510055542, -2.0818443298339844, 1.4359074831008911, 1.0026495456695557, -1.2308837175369263, 1.4412904977798462, 0.5765634775161743, -2.168206214904785, -0.5468180179595947, -0.26982563734054565, -2.0601463317871094, -0.8926233053207397, 0.3202288746833801, 0.4469722509384155, -2.0326924324035645, 1.4155064821243286, -0.15381453931331635, -0.4963310658931732, -1.7018526792526245, -0.12280857563018799, 0.9276548027992249, 0.35877758264541626, -1.443616509437561, 0.5844177007675171, 1.2273343801498413, 0.43259966373443604, -0.6179494261741638, -0.528649091720581, -0.3166425824165344, 0.3945706784725189, -2.136228084564209, 0.6841455698013306, -0.8795874118804932, 0.4717111587524414, 0.1514611542224884, 0.833380401134491, 0.33701586723327637, 0.601588249206543, 0.9360658526420593, 0.20335885882377625, -1.3464480638504028, -1.1932311058044434, -1.6460901498794556, 0.8667706847190857, -0.30476900935173035, 0.5412582159042358, 0.9593622088432312, -0.7143306136131287, -0.9093967080116272, 0.015657585114240646, -0.5544920563697815, 0.6400254368782043, 0.4800228774547577, 1.6872600317001343, 0.31755661964416504, 1.393317699432373, 0.07679618895053864, -0.8970616459846497, -0.10767481476068497, -1.6019498109817505, -0.1943766474723816, 1.084381103515625, -0.18095101416110992, -0.29126307368278503, -0.3786076605319977, -0.7197497487068176, -1.4316788911819458, 2.1287734508514404, 1.1528562307357788, -0.22929580509662628, 0.6171225905418396, 2.0368103981018066, -0.871595025062561, -0.7080643177032471, 0.1545487493276596, -1.1693289279937744, 0.35201096534729004, -1.4678977727890015, 0.848202109336853, -0.8021560311317444, -0.9698854684829712, 1.4768637418746948, -0.37038740515708923, 0.10551764070987701, -1.2733123302459717, 1.9979300498962402, -1.9803967475891113, 0.8824845552444458, -0.7424213290214539, 1.1228530406951904, -0.27104127407073975, -0.32695314288139343, -0.8649221062660217, -1.5040379762649536, 0.02070460468530655, 0.4916830062866211, -0.5799379348754883, 0.3929702639579773, 1.6251593828201294, 0.5948416590690613, 0.1457064300775528, 0.20420101284980774, 0.32168352603912354, -2.3646063804626465, 0.3066410720348358, -0.32546839118003845, -0.42355775833129883, -1.393233299255371, 0.23959894478321075, 0.083701491355896, -1.6444531679153442, 0.8512482047080994, -0.7091661691665649, 0.6963471174240112, -0.17947810888290405, -0.059070829302072525, 0.3855995833873749, -0.7100539207458496, -1.2094610929489136, 0.7535910606384277, -0.1608305126428604, -0.34129711985588074, 1.1969225406646729, -0.24276377260684967, 0.01878153719007969, 0.8339663743972778, -0.28132280707359314, -0.025294294580817223, 1.424797534942627, -0.9668728709220886, 0.572709321975708, -2.496157646179199, 0.16983072459697723, -0.995794415473938, -0.27237647771835327, 0.5042598247528076, -1.4792548418045044, -0.4450981616973877, -0.6490049958229065, -0.2701474726200104, -0.1285112351179123, -0.43196722865104675, -0.45647189021110535, -0.031203072518110275, 0.8581725358963013, -1.9133076667785645, 0.6312763690948486, 1.2223811149597168, 1.3959368467330933, 0.883366048336029, -0.8743641376495361, -0.8349840044975281, 0.9244630336761475, -1.2415826320648193, -1.4639273881912231, -0.6062346696853638, 0.6128938794136047, 0.6744053959846497, 1.3208597898483276, -0.7169403433799744, 1.1857514381408691, -0.4701070487499237, 0.7895348072052002, 0.9106235504150391, -0.4933842420578003, -1.44131600856781, 0.007872438058257103, 0.5084673762321472, 1.8204858303070068, -0.008891163393855095, 1.606430172920227, 0.6913145184516907, 0.09177237004041672, 0.43373891711235046, 0.4296130836009979, 0.12771432101726532, 0.2315933257341385, -0.6180892586708069, -0.5043414831161499, 0.4336695075035095, -1.2130295038223267, 0.19016802310943604, 0.6150248646736145, 1.4334453344345093, 0.07004868984222412, -0.4867277443408966, -2.1598823070526123, 0.05725792050361633, -0.14034222066402435, -0.8158409595489502, -0.776307225227356, 0.5779224634170532, 0.4129810631275177, 2.2669472694396973, -0.9518476724624634, 1.8930213451385498, -0.48330482840538025, -0.03142377361655235, -0.4898747503757477, 1.147784948348999, 0.133233904838562, -1.3749501705169678, -0.8074237108230591, 0.6949084997177124, -0.9289232492446899, 0.515871524810791, -0.7585099935531616, -1.5985641479492188, 0.3922646641731262, -0.7048566937446594, 1.9275856018066406, 0.5604162812232971, 0.3193317651748657, 0.2924412488937378, -1.1746419668197632, 0.2428695112466812, 0.4632614254951477, -0.19677941501140594, -0.942044734954834, -0.3188137114048004, 1.6885261535644531, -0.7717347145080566, 1.4220011234283447, -0.510017991065979, -0.8042749166488647, 0.5917132496833801, 0.4570298492908478, -0.6893635988235474, -1.5604444742202759, -0.9576707482337952, 1.252320647239685, 0.1232566237449646, -0.15764126181602478, -0.7132253050804138, -0.30735543370246887, -2.0359158515930176, 0.6573584079742432, 1.1967716217041016, -0.7721946835517883, 0.9178104996681213, -0.3720979392528534, 0.5238752961158752, 1.1880364418029785, 0.3540361821651459, -0.6335596442222595, 0.8297712802886963, -2.1201367378234863, 0.6653119921684265, -0.9836238026618958, 3.389629364013672, -0.7081546187400818, -0.8638523817062378, -0.4536583423614502, 0.06935945153236389, 2.138183116912842, -1.2801964282989502, 1.6308643817901611, 0.8439164161682129, -0.025669291615486145, -0.5781439542770386, 0.6967278718948364, 0.5272228717803955, -0.23799251019954681, -1.3888776302337646, -1.276695966720581, -0.5293793678283691, -1.9369889497756958, -1.9088488817214966, -2.8266499042510986, 0.6555210947990417, -1.1216973066329956, -1.142368197441101, -1.1805285215377808, 0.46205267310142517, -0.55757075548172, 0.8505668044090271, 0.6401589512825012, 0.7023569345474243, 1.3050463199615479, -0.8998906016349792, 0.48744750022888184, 0.7416771650314331, 1.3774813413619995, 0.31211361289024353, -0.7020406126976013, 0.18185310065746307, 0.7004386782646179, -0.839455246925354, -1.187113881111145, 0.46756264567375183, -1.3021876811981201, -0.5825977921485901, -0.5422713756561279, 1.648370385169983, -1.647627353668213, -1.2437353134155273, 2.518709182739258, 0.4406755864620209, 0.6078804135322571, 0.729067325592041, 0.4088970124721527, 0.7194890379905701, -0.5621077418327332, 0.11134946346282959, -1.3837250471115112, 2.1880102157592773, -0.6887124180793762, -1.2499362230300903, -0.5943945050239563, -1.3336135149002075, 0.7984803318977356, 0.43592554330825806, 0.5352944731712341, -1.1175789833068848, 0.8578688502311707, 1.2445873022079468, 1.543907880783081, -0.3055008351802826, 0.426171213388443, 1.0120246410369873, 1.5348453521728516, 0.5629629492759705, -0.9752320051193237, 1.026714563369751, 0.03258895501494408, -2.027175188064575, 1.3030965328216553, 0.8317315578460693, 0.13578227162361145, 1.046244740486145, -0.003964587114751339, 0.8561106324195862, 0.7132399678230286, -0.6081644296646118, -2.5345427989959717, 1.5403969287872314, 1.732241153717041, 0.22913718223571777, 0.111876480281353, 0.6188908815383911, -0.048641156405210495, 1.0213629007339478, 0.39184245467185974, -0.6286125779151917, 0.5847413539886475, 0.5348641872406006, -1.240303635597229, -0.3217993974685669, -0.716961681842804, -0.1390872746706009, 0.5757394433021545, 0.12126023322343826, 0.21162503957748413, 0.5757143497467041, 0.3918326795101166, 1.143305778503418, -1.4033641815185547, 1.202777624130249, -2.5497498512268066, 0.39406657218933105, 1.5357986688613892, -0.23069119453430176, 1.3817942142486572, 0.5921380519866943, 0.27250945568084717, -0.6718708872795105, -0.7125983834266663, 1.415714979171753, 0.9975833296775818, -1.4926990270614624, 0.7285737991333008, -1.3998042345046997, -0.2826768755912781, 0.48505085706710815, -0.2196253538131714, -1.4199583530426025, 0.7850315570831299, -2.1137382984161377, 1.1401444673538208, 0.28051066398620605, 1.3683781623840332, 0.35029762983322144, -0.27974212169647217, 0.7450258135795593, 0.8820900917053223, -0.35660722851753235, 0.7496739029884338, -1.242592692375183, 2.133939027786255, 0.8715195655822754, -0.5797992944717407, -0.8922671675682068, -0.27477291226387024, 1.1499359607696533, -0.7820935845375061, -0.5473063588142395, -1.307523250579834, -1.886523962020874, 0.4113028049468994, -0.3768748939037323, 0.8380845785140991, -1.0903998613357544, -0.27422553300857544, 1.1700644493103027, -0.8167003989219666, -0.00870431587100029, -0.25047567486763, -0.577693521976471, -0.8822503685951233, 1.7568235397338867, -0.7355344295501709, -0.5963753461837769, -0.9930616021156311, -2.0754897594451904, -0.2168162763118744, 0.4694770574569702, -0.19968454539775848, -0.08612826466560364, 0.8825541138648987, -0.7176868915557861, 0.019435331225395203, -0.5783736109733582, 1.6106435060501099, -1.6206854581832886, 0.21654000878334045, -0.5637560486793518, -0.06381304562091827, -0.08401044458150864, 0.2606777548789978, 0.9206234216690063, 1.4014259576797485, -1.327865719795227, 1.1198724508285522, -0.9367362260818481, -1.3750853538513184, 0.0782487615942955, 1.2289475202560425, 1.6740564107894897, 0.026176465675234795, 1.5854967832565308, -0.10170810669660568, -0.6514601707458496, -1.6362769603729248, -0.2470831423997879, -1.8393241167068481, -0.45607081055641174, 1.4902825355529785, 0.20847193896770477, 0.07755687832832336, -1.4461109638214111, 0.4350898563861847, -0.6214316487312317, 0.5277040600776672, 0.041134074330329895, -0.9169013500213623, -1.5331099033355713, -0.7579681277275085, 0.7643961310386658, 1.0995632410049438, -0.3616287410259247, -0.36550837755203247, -1.2503712177276611, -0.6954047679901123, 0.5405731797218323, 0.4935104548931122, -1.4105807542800903, -1.05596125125885, 0.43867677450180054, -0.7614096403121948, 0.6150473952293396, -1.0182912349700928, -1.761443018913269, 0.13658644258975983, -0.04858586937189102, 1.7774405479431152, 0.3934374451637268, -1.299978256225586, -0.032866235822439194, -0.7302518486976624, -0.6856483817100525, -0.06744366884231567, -1.3712868690490723, 1.5951881408691406, -0.04947274923324585, 1.2501412630081177, -0.621772289276123, -0.11271506547927856, 0.18916714191436768], [0.4886620044708252, 1.2373783588409424, -1.1221171617507935, -0.6455571055412292, -0.5155606865882874, -0.6172462701797485, 1.020079255104065, 2.8023805618286133, 1.7774301767349243, -1.2909159660339355, -0.9429960250854492, 0.1463783085346222, -0.3324190378189087, -0.4837368130683899, 0.15527142584323883, -0.849084734916687, 0.6168181300163269, -0.5029580593109131, 0.15577413141727448, 0.2563436031341553, -1.523573398590088, -0.10675712674856186, 0.572664201259613, 0.31947436928749084, 1.0048185586929321, 1.718961477279663, -0.02077937126159668, 0.36547088623046875, -0.23384976387023926, -1.0993661880493164, 0.9544039368629456, 0.17146095633506775, 0.6796201467514038, 1.1253832578659058, -1.1443774700164795, 0.493821382522583, -1.1935890913009644, -0.5235714316368103, 1.5425320863723755, -0.3850267231464386, -1.7540295124053955, -1.846848726272583, -0.8951415419578552, 0.2688710391521454, -0.41079196333885193, 0.39970093965530396, 2.024674892425537, -0.39590343832969666, 0.07118909806013107, 1.2265487909317017, -1.174472451210022, -0.1141701489686966, 0.04755022004246712, 0.12878212332725525, -0.8156623244285583, 0.1985713392496109, -1.1569011211395264, -0.6668881773948669, -1.6729964017868042, -0.26326680183410645, -1.7354929447174072, -0.6404898166656494, -0.3482990264892578, -0.060637619346380234, -0.4614523649215698, -1.1917799711227417, -1.7519173622131348, -0.7053726315498352, -0.9807476997375488, 1.2283183336257935, 0.4839481711387634, -1.727715015411377, 0.10175561159849167, -0.09738010913133621, 0.24708952009677887, -0.9513948559761047, 1.5965584516525269, -0.6368820071220398, 2.3356757164001465, 0.8248937726020813, 1.533865213394165, 0.6116081476211548, 0.8341912031173706, 1.267652988433838, 0.37392377853393555, -1.2861193418502808, -0.38786786794662476, 0.6758680939674377, 0.3913545310497284, 0.9721869826316833, -1.6496222019195557, -1.8409404754638672, 0.26585954427719116, 1.286803960800171, -0.5143696665763855, 0.0617542639374733, -0.794194757938385, -0.3608684539794922, 1.8706178665161133, 0.8339549899101257, 0.08602064847946167, 1.2221318483352661, 0.4071162939071655, 0.5540568828582764, 0.2693904936313629, -1.5662641525268555, 1.4996442794799805, -1.0735584497451782, -0.7828165888786316, -0.328024685382843, 0.8729740381240845, 1.657076358795166, -0.27028220891952515, -2.0112736225128174, -1.1012133359909058, -0.05731769651174545, -0.17692656815052032, -0.5937812328338623, -0.745310366153717, -1.2207634449005127, 0.06893637776374817, -0.5397018790245056, -1.8482489585876465, -0.04892125353217125, 0.18529635667800903, 0.9029759168624878, 0.10088691860437393, 0.8007851839065552, -1.0602284669876099, 0.24581706523895264, 0.06622522324323654, -0.018898673355579376, -2.121161460876465, 0.5282394886016846, -0.16277676820755005, 1.0136314630508423, 0.7309884428977966, -1.9115592241287231, 1.2696199417114258, 1.6103423833847046, 0.6342934370040894, -0.3899780511856079, -0.6277637481689453, 1.4677311182022095, -0.28921422362327576, -0.24689067900180817, 1.822514295578003, -1.3186694383621216, -0.13607893884181976, -0.24383695423603058, 0.8752738237380981, -0.7059857249259949, -0.06001882627606392, -0.2247799187898636, -0.12415643781423569, 0.7505007982254028, -0.36041200160980225, -1.588315486907959, -0.5528258085250854, -0.03420378640294075, -2.500892400741577, -0.44325146079063416, -0.5926591157913208, 2.118741750717163, 0.9132187366485596, 0.8936455249786377, -0.4187803864479065, -0.5077986121177673, -0.5312157273292542, 0.6905195116996765, 0.5159799456596375, -0.16013793647289276, 0.7405728697776794, 0.5265486836433411, 1.7853399515151978, 0.4065237045288086, -0.8152157068252563, -1.44856858253479, -1.4524058103561401, -0.8013659715652466, 1.366295337677002, 0.7303661704063416, 0.6509993076324463, 0.0015686380211263895, -1.347080111503601, -0.41045019030570984, -0.20626375079154968, 0.6130643486976624, 0.03758014366030693, 1.2966068983078003, 1.402433156967163, -1.3729761838912964, -0.40424656867980957, 2.1071836948394775, 0.2533866763114929, 0.47416597604751587, 0.46679922938346863, -0.9072666168212891, -0.4325196146965027, -0.8864132761955261, -0.9718453884124756, -1.4032281637191772, 1.3876765966415405, 0.23919090628623962, 0.6871858239173889, 0.6740167140960693, 0.8151721954345703, 1.0114305019378662, 0.43163904547691345, 1.3877637386322021, 2.0188534259796143, -1.286740779876709, -0.7886798977851868, 0.012092141434550285, -1.1907033920288086, 1.2102248668670654, -1.7418891191482544, 0.199382945895195, 0.8303630352020264, -0.08282309025526047, -1.0952597856521606, 1.7790579795837402, 0.8743270635604858, -0.28646162152290344, -0.15786731243133545, 0.594609260559082, -2.0351107120513916, -0.7923629879951477, 0.02101505920290947, 1.3071255683898926, -1.3471182584762573, -0.2735283672809601, 0.5224502086639404, -0.9125422835350037, 0.4811134934425354, -2.1098644733428955, 0.6500421762466431, -0.15772785246372223, 0.3565530478954315, 0.9361696839332581, -1.9751712083816528, -1.453589677810669, -1.2406394481658936, -1.1509290933609009, -1.3711779117584229, 0.7106451988220215, -1.955541729927063, -0.30735158920288086, 0.9146917462348938, 0.12533824145793915, 0.15006794035434723, -1.3339499235153198, -1.2261662483215332, 0.5087785124778748, -0.8129361271858215, -1.0559812784194946, 0.05513763055205345, -1.8539056777954102, 0.5438493490219116, 0.8356572389602661, 0.4505029022693634, 0.12738797068595886, -0.26528796553611755, 0.2489674836397171, 0.2791420817375183, 0.6466802954673767, -1.3076250553131104, 0.07584977895021439, 0.800015389919281, -0.6315901279449463, 0.5021528601646423, 0.7677383422851562, 0.5657541751861572, -2.018402338027954, 0.22426682710647583, -0.2924976944923401, -1.4744093418121338, 0.3260902762413025, -0.3593769669532776, 0.01125163771212101, -0.5009011030197144, -1.2876224517822266, 0.13626442849636078, -0.9579139947891235, -0.3915100395679474, -0.0320151150226593, -0.653065025806427, -0.35620006918907166, 1.277448058128357, -1.094802975654602, -0.9359816312789917, -0.6829652190208435, 1.2608189582824707, 0.4592677056789398, -0.6792193055152893, 1.3012487888336182, -0.1749725490808487, -0.8954140543937683, 0.27025943994522095, 1.130527138710022, 0.09884964674711227, -1.01460599899292, 0.9972960948944092, -0.1749522089958191, -2.5397629737854004, -0.8278694748878479, 0.08227550983428955, 0.2547267973423004, 0.500977635383606, -0.008975652977824211, 0.43891647458076477, 2.0742430686950684, -0.12969809770584106, -1.3017075061798096, 0.16798242926597595, 0.33497628569602966, 1.6181472539901733, -1.8013852834701538, -0.5098472833633423, -1.4411389827728271, -0.5547093152999878, -0.9271537661552429, -1.9279258251190186, -0.7078654170036316, -0.7697893381118774, 0.9380834698677063, -0.29760465025901794, -0.0040395851247012615, -0.345829039812088, 0.829766035079956, 0.5812879204750061, 0.5921744108200073, 0.06133376806974411, -0.8515698909759521, -0.39682310819625854, 1.1366292238235474, -0.3199138939380646, 0.27402135729789734, -0.5271604061126709, -0.002483647083863616, 0.2763848900794983, 1.561921238899231, 2.346773624420166, -0.558562159538269, 2.202284336090088, -2.1264889240264893, 0.10495809465646744, 0.4753783345222473, -1.469283103942871, -0.3234921395778656, 1.0069990158081055, -0.1089649423956871, 1.4746835231781006, 0.4505723714828491, 0.7281728982925415, 0.8996440172195435, 0.16170717775821686, 0.16494394838809967, 1.8260493278503418, -1.209116816520691, 0.32757025957107544, 1.6471152305603027, -0.7456294894218445, -0.6780688762664795, -0.28320494294166565, -0.7826175093650818, 0.7168095707893372, 0.6848483085632324, 1.7300033569335938, -0.38795164227485657, 0.040478043258190155, 0.6833316683769226, 1.0265685319900513, -0.5425342917442322, -1.0747052431106567, -1.1317691802978516, -1.29972243309021, -2.516005039215088, 0.9848423600196838, 0.1986169070005417, -1.3875253200531006, 1.1073682308197021, -0.6573209762573242, 2.6103484630584717, 0.9673234820365906, -1.2114553451538086, 1.2901257276535034, -0.17922477424144745, 0.9275996685028076, 0.723451554775238, -0.48082318902015686, 1.5154445171356201, 1.1495710611343384, 0.31024837493896484, -1.0376112461090088, -0.21331755816936493, -1.68561589717865, -0.6345192790031433, 0.4627578556537628, 0.16109268367290497, -1.7321345806121826, -0.7025920152664185, -1.035197138786316, -0.8941055536270142, -1.1341640949249268, -0.8710625171661377, 1.6552060842514038, 2.211517572402954, -2.670092821121216, -0.9136064052581787, -2.3692400455474854, 0.019006531685590744, 0.16485291719436646, 0.37532296776771545, -1.9549095630645752, 0.3108218312263489, -1.9719924926757812, 1.0365570783615112, 0.04434790089726448, 2.342388153076172, -0.23091918230056763, 1.2840083837509155, 0.4603974223136902, -0.6403814554214478, -0.28122907876968384, -0.9123727679252625, 0.2170715034008026, -0.1366988867521286, -0.9518870115280151, 0.9553251266479492, -0.23976212739944458, -0.9787387847900391, -0.3063512146472931, 1.9557089805603027, -0.33230799436569214, -0.947266697883606, 0.41349557042121887, 0.2087051421403885, 1.0385812520980835, -1.2986866235733032, 0.5354254245758057, -0.1192973405122757, -0.4556412100791931, -1.0032376050949097, 0.48189622163772583, -0.6627432703971863, 0.7291958928108215, -1.122646450996399, 1.5194188356399536, -0.7990618348121643, -0.21569949388504028, -1.0234755277633667, 0.41519036889076233, -0.6556268930435181, 1.3937581777572632, 0.1856526881456375, -0.6348536610603333, 0.45723116397857666, -0.4052850604057312, 1.242146372795105, -0.9428775906562805, 0.7025040984153748, 0.29054224491119385, -1.0050954818725586, 0.6576661467552185, -0.14427320659160614, 2.248616933822632, -0.24475371837615967, -0.1772313117980957, -0.29402199387550354, -0.4626292586326599, 0.6659603714942932, 1.7237038612365723, -1.1325414180755615, -1.1408599615097046, -0.06325782090425491, -0.09463919699192047, 1.394163727760315, 0.2620927095413208, -0.13331854343414307, 0.7557055354118347, 0.5583254098892212, -0.007861644960939884, -0.5546872615814209, -0.753720223903656, 0.9492325782775879, -1.2768319845199585, -1.182108759880066, 1.9875986576080322, 0.04389290511608124, 0.30628782510757446, 0.8921061754226685, 0.07138184458017349, -2.115812063217163, 0.05077561363577843, 0.1670026332139969, 0.5511754751205444, 0.4625595808029175, 0.1612451821565628, -0.30864065885543823, -1.551184892654419, -1.506335973739624, 0.40024998784065247, 1.5834602117538452, 0.7204193472862244, -0.9978665709495544, 0.0967152938246727, -1.8619563579559326, -0.6673175096511841, -0.4821977913379669, 0.2968115508556366, -0.08682549744844437, -1.463516116142273, -2.136526584625244, -0.043075576424598694, 0.7847617268562317, 0.9508548974990845, 0.1782870888710022, -0.23668283224105835, -1.0421744585037231, -1.0176713466644287, -0.028238577768206596, -1.1032339334487915, -1.5900272130966187, -0.46360310912132263, -0.3248904347419739, -0.582516610622406, -0.1131599023938179, -1.4751455783843994, -1.0772323608398438, 0.6602991223335266, -1.2823035717010498, 1.2151292562484741, 2.3155529499053955, 1.617267370223999, -0.08913663774728775, 0.6191099286079407, 0.36695078015327454, -1.1054080724716187, 1.3144798278808594, 0.5323922038078308, 0.6521885395050049, -0.3819107115268707, 0.20937153697013855, 0.4097229838371277, -1.1275994777679443, 0.8825390338897705, -0.4192761182785034, 0.5262070894241333, -0.018993409350514412, -0.30441781878471375, -0.07991954684257507, 2.0339438915252686, 0.7277030944824219, -1.3029415607452393, 2.057934284210205, 1.672692894935608, 0.6564204692840576, -0.7140809893608093, -0.6521497964859009, 0.5486971139907837, -0.49573472142219543, -1.043860673904419, 0.39726588129997253, 1.221286654472351, 0.7488006353378296, 0.5417400598526001, 1.3563823699951172, 1.3474035263061523, 0.27803322672843933, -0.7550107836723328, -1.3290308713912964, -0.34544578194618225, 0.5089046359062195, 0.6285994052886963, 1.0058138370513916, -0.6152191758155823, -1.5248949527740479, 0.995047926902771, -0.4999598562717438, 0.2604183852672577, -0.004595134872943163, 0.6166563630104065, -0.5349335670471191, -0.26535215973854065, 1.7661402225494385, -0.7905176281929016, 0.997230589389801, 0.5775213241577148, -0.6338788270950317, 0.44835156202316284, 0.6012658476829529, -0.036013342440128326, -0.058758094906806946, -1.5254149436950684, -0.40136533975601196, -0.3806670606136322, 0.9090564250946045, -0.514508068561554, 0.05164262652397156, 0.24522067606449127, -0.1812831163406372, -1.0292277336120605, -0.7937528491020203, -0.4625576436519623, 0.006412144750356674, -2.173077344894409, -0.903037965297699, 0.1771245300769806, -0.18086843192577362, 0.4606722891330719, 0.3737366497516632, 0.6840994358062744, -0.7715305089950562, -0.5224931240081787, 1.276413917541504, 1.571663498878479, -3.1673295497894287, 1.15299654006958, 0.8709570169448853, 0.15464745461940765, 1.5078747272491455, -1.1967169046401978, 0.8093018531799316, -1.095961093902588, 0.5758183002471924, 0.7328081130981445, -0.05155154690146446, 0.1727273017168045, -1.2845286130905151, 1.779642105102539, -0.19143667817115784, -1.3566861152648926, 0.23676303029060364, 0.34300050139427185, 0.7771636247634888, -0.05054166913032532, 1.7017158269882202, 0.34273359179496765, -1.3741415739059448, 1.343937873840332, 0.9058878421783447, -0.17745341360569, -0.5501757860183716, -0.5114045739173889, 0.7557534575462341, -1.6265389919281006, 3.291041374206543, 2.3477203845977783, -0.027333540841937065, 0.040905892848968506, 0.9142912030220032, 0.7427994608879089, -1.4325703382492065, -0.4569593667984009, -0.9824963212013245, -0.5322731137275696, -2.3826754093170166, -1.0572744607925415, -0.7553338408470154, 0.07800254970788956, -0.8571673631668091, 0.8359310030937195, 1.4306217432022095, 0.19589556753635406, -1.1785591840744019, 0.5836473703384399, 0.33763328194618225, 0.22281043231487274, 0.5450670123100281, 0.7265572547912598, -0.9184137582778931, 2.378171920776367, 0.4380037784576416, 1.760953426361084, -0.46225762367248535, 0.4546862840652466, 0.4232633709907532, 1.1952834129333496, -0.6676384210586548, 0.25912410020828247, -0.962562620639801, 0.2948581874370575, 0.738523006439209, 0.7253516316413879, 0.9207658767700195, 0.05849330499768257, -1.00295889377594, 0.1104140356183052, -2.2172539234161377, 0.5651311874389648, 0.3114209771156311, 0.2675090730190277, 0.28934919834136963, -1.1384060382843018, 1.080694317817688, -0.21937629580497742, 0.9763188362121582, 0.19800759851932526, 0.49350884556770325, 1.322483777999878, -0.5608869791030884, 0.4901181757450104, -0.37171438336372375, -0.1588129848241806, -1.8385162353515625, 0.2745303213596344, -1.6179487705230713, 0.7778792381286621, -0.4973594546318054, -0.9680836200714111, 0.4103948771953583, 0.4568334221839905, -0.0435846671462059, -0.6231120824813843, -1.3201457262039185, 1.2011536359786987, 0.5268506407737732, 0.41050273180007935, -2.0738306045532227, 2.407163619995117, 1.2418359518051147, 0.5040061473846436, 0.12622874975204468, 0.05894806608557701, 0.4550929665565491, -0.8919251561164856, 2.3609230518341064, 0.192566379904747, 1.4411050081253052, -0.04123380035161972, 0.1696484237909317, 0.8899128437042236, 0.7198711037635803, 0.18712203204631805, -0.12680453062057495, 0.3038291037082672, 1.4682111740112305, 0.4016405940055847, -1.0939587354660034, -1.4877346754074097, -0.8236867189407349, 0.3662266433238983, -0.49492403864860535, -0.6377578377723694, 1.7314403057098389, -0.7320138216018677, 1.7229238748550415, -2.9833569526672363, 1.4155433177947998, 0.4185890555381775, 1.876281976699829, -0.6865960359573364, -0.5228578448295593, 0.6352408528327942, -0.8130148649215698, 0.23541826009750366, -1.233102560043335, -0.45455560088157654, -0.35018032789230347, 1.3213666677474976, 1.3260771036148071, -0.35010337829589844, 1.0649055242538452, 1.4144121408462524, -0.6168816089630127, -0.8398080468177795, 0.18326979875564575, -0.803570032119751, -0.212962806224823, 0.24969995021820068, 0.1764189600944519, -0.12323670089244843, 1.5920695066452026, -1.324101448059082, -1.0758224725723267, 0.17942458391189575, 0.7853977680206299, -0.08025746047496796, 0.716937243938446, 1.154383897781372, -0.3398840129375458, 0.3432409465312958, 0.22508667409420013, 0.8294188380241394, 1.1535372734069824, -1.2359172105789185, 0.6910799145698547, -0.08640885353088379, -0.15045863389968872, -0.09793014824390411, 0.7534011602401733, -0.4682510793209076, 0.18860791623592377, 0.9610279202461243, -1.224308729171753, -1.3185166120529175, 0.2627894878387451, 0.6549131274223328, -1.299976110458374, 0.8477907776832581, -0.7538439035415649, 1.5027602910995483, -0.8773713707923889, 1.8218238353729248, -0.46327897906303406, -0.5481511354446411, 1.6753290891647339, 0.39564767479896545, -0.8404079079627991, 0.9692628979682922, -0.013584881089627743, -1.0834267139434814, -1.2187546491622925, 0.9191250801086426, -0.6819795966148376, 0.16655348241329193, 0.2282951921224594, -0.11631334573030472, -0.1585305780172348, -0.43882787227630615, -0.08900380879640579, 0.15589743852615356, -0.5853797197341919, -0.2866651117801666, -1.0883668661117554, 0.20246058702468872, 0.5262740254402161, 0.47560036182403564, 0.45333510637283325, -0.04685454070568085, 1.6877083778381348, 0.6421282887458801, 1.327130913734436, -0.5129064917564392, 0.47101715207099915, 0.32511770725250244, -0.04194021224975586, 0.5594338774681091, -0.7960295677185059, 1.2189234495162964, -0.6474090218544006, 0.8118467926979065, -0.031255170702934265, 0.9936190247535706, 0.1541939675807953, 1.9927077293395996, -1.3682273626327515, 2.2405502796173096, -1.1089614629745483, -1.2245506048202515, -1.1222906112670898, 0.3573349118232727, -0.7969629168510437, -1.0766433477401733, -0.4619927406311035, 0.4746171534061432, -0.16266444325447083, -1.7817356586456299, 0.868615984916687, 0.29054251313209534, -1.9849989414215088, 0.861995279788971, 0.10140452533960342, 0.29581722617149353, -1.2229565382003784, 1.7642894983291626, 0.26509353518486023, -1.6799883842468262, -1.5958651304244995, 0.527874767780304, -0.28489163517951965, 0.004255587235093117, 0.5837909579277039, -0.2897597849369049, -0.039450645446777344, 0.03305425867438316, 1.94425368309021, 0.7553611993789673, 0.2540205419063568, 0.14053300023078918, -0.526257336139679, 1.6096875667572021, -0.5393150448799133, -1.0373209714889526, -0.10790420323610306, 0.2592477798461914, 0.4706778824329376, -0.5444043278694153, -0.3235677480697632, -0.3536470830440521, -0.8944879174232483, -0.7283465266227722, -1.2006022930145264, 0.7223048806190491, 0.5203739404678345, -0.6554831266403198, 1.4444646835327148, -1.314526915550232, -0.4439370334148407, -0.051957931369543076, -1.325107455253601, -0.8932751417160034, 1.0125813484191895, 0.7421321272850037, -0.5112831592559814, 0.34367039799690247, -0.5853655934333801, 1.1508498191833496, 0.0006462465389631689, -1.6192644834518433, -0.3405154049396515, -0.13295017182826996, -1.0787814855575562, -1.5771828889846802, -0.2639499306678772, -1.8059697151184082, -0.5880808234214783, -0.6181902289390564, -0.808394193649292, 1.2595051527023315, 0.7907235622406006, -1.3019194602966309, -1.383123517036438, 0.8822139501571655, 1.1198738813400269, -0.42628899216651917, 0.8768798112869263, -0.6972255706787109, 0.5097151398658752, -1.1685385704040527, -0.10420437157154083, 0.1882963627576828, -0.9947013258934021, -2.3180246353149414, -0.429479718208313, -1.0660072565078735, 2.8721725940704346, -1.1308783292770386, 0.5873086452484131, 0.6327440738677979, -1.48505437374115, 0.154174342751503, -2.0901620388031006, 0.20105673372745514, 1.4711309671401978, 0.09143079817295074, -0.17640623450279236, 0.27306005358695984, 0.537562370300293, -0.04112403467297554, 0.03035036288201809, 0.819925844669342, -0.642084538936615, 0.4139549136161804, -1.3718844652175903, 0.3346925675868988, 0.03324482962489128, -1.9308454990386963, 0.23549258708953857, 1.705748438835144, -0.06835897266864777, -1.409883737564087, 0.9621841311454773, 1.1660048961639404, -0.01454066764563322, -0.8269394040107727, -1.0755141973495483, 2.223256826400757, 0.02697598934173584, 1.957100510597229, 0.7004036903381348, -0.504427433013916, -0.1851256787776947, -0.5720210671424866, 0.02377857081592083, 0.026366742327809334, 1.9409608840942383, 0.8968749642372131, 0.7964062094688416, 0.39996466040611267, 1.1095398664474487, -0.6376980543136597, 0.25812283158302307, 0.36246898770332336, 0.9316895008087158, -0.6963679790496826, -0.6641314029693604, -1.9773520231246948, 0.09317071735858917, 0.50258469581604, -1.6316685676574707, -1.198547124862671, 2.4285500049591064, 0.018261069431900978, -1.3141894340515137, -1.0789319276809692, -0.4869299829006195, 1.8244110345840454, 0.35704079270362854, -1.9500377178192139, -0.2789909541606903, 0.13631126284599304, 1.7499457597732544, 0.1902770698070526, 1.1036752462387085, -0.5837951898574829, 0.10473302006721497, -1.1581006050109863, -1.9645591974258423, -2.253734588623047, 0.4094390869140625, 0.49458208680152893, 0.15384119749069214, 1.609874963760376, -0.46431779861450195, 2.048779010772705, -0.7812398076057434, -0.49977514147758484, -0.1960943192243576, 0.5874490737915039, 1.0417349338531494, -1.371071696281433, -0.8107260465621948], [1.1310921907424927, 0.009931824170053005, -0.4631166458129883, -0.716002345085144, 0.8804686665534973, -0.6437165141105652, 0.7415657639503479, -1.3542433977127075, 0.6413547396659851, 0.7400898933410645, -0.8423103094100952, 1.144709587097168, 1.8882837295532227, -1.5190234184265137, 0.09110038727521896, -0.3029922842979431, -0.4123404324054718, -2.1461477279663086, 0.6236770749092102, 0.7345372438430786, -0.42615610361099243, 1.4781557321548462, -0.7261381149291992, -0.9734130501747131, 0.7423352003097534, 0.6116517186164856, -0.7029879689216614, 1.5872514247894287, 0.1835770457983017, 1.745671272277832, -0.5131771564483643, 0.0562061071395874, -0.9842633008956909, -0.4716548025608063, 0.35641053318977356, 0.6921820044517517, -0.8724819421768188, -3.4381792545318604, -0.8029898405075073, -1.4792531728744507, -0.1296665519475937, 0.709317147731781, 0.16179442405700684, 0.11927192658185959, -0.1081402450799942, 0.7466625571250916, 0.25618916749954224, -1.1045295000076294, -0.9240484237670898, -1.2005829811096191, -0.7573277354240417, -1.1777015924453735, 0.5128252506256104, -0.002167940139770508, -0.050503235310316086, -1.9328055381774902, -1.8524980545043945, 1.3218152523040771, -0.3906245827674866, -0.18100102245807648, 0.9047690629959106, 0.8850228190422058, 0.061113931238651276, -1.4336565732955933, 0.890217661857605, -0.40025630593299866, -1.1439509391784668, 1.209115743637085, 0.8056207299232483, -1.26799738407135, -0.5529962182044983, -0.6177693605422974, -1.6718708276748657, -2.069354295730591, 0.7802784442901611, -0.49346959590911865, -0.6619521379470825, -1.2419469356536865, -0.7397054433822632, 1.1159151792526245, -0.19704987108707428, -1.2101014852523804, -0.2947039306163788, 1.8829419612884521, -1.5063320398330688, 0.3915199637413025, -0.6813006401062012, 0.8323783278465271, -1.0641117095947266, 0.34883740544319153, -1.6983745098114014, 0.28069108724594116, 0.6582183837890625, 1.3202966451644897, -0.8090871572494507, 2.3683207035064697, 0.9199545979499817, 0.18325774371623993, 1.1821449995040894, -0.08237037807703018, -0.5361550450325012, -2.266050338745117, -1.7138444185256958, 0.4762166440486908, 0.7899144291877747, 0.6108425855636597, 0.3993763029575348, 0.20339056849479675, 0.6169979572296143, 0.19569066166877747, 0.678719699382782, -1.1418699026107788, -0.2746961712837219, -1.4822890758514404, -1.1997895240783691, -0.36161795258522034, 1.0463217496871948, 0.4784499406814575, -0.050389256328344345, 1.2346055507659912, -1.7515068054199219, -0.3670056164264679, 0.3499780595302582, -2.346057653427124, 0.8453247547149658, 0.3878189027309418, 1.2611089944839478, 1.0544681549072266, 0.26864051818847656, 0.4081423878669739, 1.0079829692840576, 0.7911583185195923, 1.1443758010864258, 2.543945789337158, -1.1540836095809937, 1.0957404375076294, 0.22701871395111084, -0.2419302761554718, 0.3281058073043823, -0.4912799298763275, 0.04097026214003563, 0.14851583540439606, -1.44882071018219, 0.6148869395256042, 0.8828767538070679, -2.2694454193115234, 0.7583065032958984, 0.07921262085437775, -0.19364799559116364, 0.5541314482688904, -0.5409358143806458, -1.8903192281723022, 0.10020850598812103, 1.034199833869934, 0.02620100975036621, -0.14888478815555573, 0.6165611147880554, 0.11051319539546967, -1.4234765768051147, 0.7202286124229431, 1.5351347923278809, 0.5290836095809937, 0.29774653911590576, -1.4201178550720215, 0.5932700037956238, -0.25490570068359375, 0.2811620235443115, -2.2285330295562744, 0.018884262070059776, 1.1488890647888184, -0.9804245233535767, 0.1990041434764862, -0.5923202037811279, -2.6891469955444336, -1.6506679058074951, -0.6810911297798157, 0.032514650374650955, -0.930846631526947, 0.3895147442817688, -0.5783060193061829, -0.7715778946876526, 0.5786795616149902, 0.3545343279838562, -1.4641491174697876, 0.18091458082199097, -0.8283532857894897, -0.8616220951080322, -0.1465577781200409, 0.0360482819378376, -2.084484338760376, -2.1321330070495605, 0.2493305504322052, -2.71628475189209, 0.03548392653465271, -1.039977788925171, 0.019172603264451027, 1.5835727453231812, -1.0666868686676025, 0.5717054009437561, -2.265611171722412, 1.1936384439468384, -0.5783049464225769, 0.15944790840148926, 0.9431141018867493, 0.9640935659408569, 0.5880153179168701, 1.8649336099624634, -1.3923656940460205, -0.381664901971817, 0.20680943131446838, 1.6726362705230713, -0.056381091475486755, 0.4640117585659027, 0.2859202027320862, -1.682515263557434, -1.7683284282684326, 0.23728807270526886, -0.853895366191864, -0.9574194550514221, 0.9522666335105896, -0.44619593024253845, 0.3961399793624878, 0.37959492206573486, 0.32487425208091736, 1.5208547115325928, 1.9646943807601929, 0.8066357374191284, 1.4031919240951538, 0.3231637179851532, 1.1614999771118164, -1.265592336654663, -0.5126279592514038, -1.1727697849273682, -1.5196930170059204, -0.8805846571922302, 1.659676432609558, -0.3881470263004303, 0.6294472813606262, 0.37055137753486633, -0.3855209946632385, 0.6450128555297852, -1.117157220840454, -0.5169218182563782, -1.020112156867981, 0.0951886773109436, 0.07387660443782806, 0.01910654827952385, -0.7994641065597534, -1.194711446762085, -0.3577776551246643, 1.7927286624908447, 1.2675001621246338, -0.04893294721841812, -0.04162803664803505, -0.21288427710533142, -0.1584683507680893, -0.009042718447744846, -0.13938403129577637, -0.002085554413497448, -0.32712048292160034, -0.5289013385772705, -2.257450819015503, 0.3058789372444153, 0.17123469710350037, -0.5409228205680847, -0.36793431639671326, 1.0730220079421997, -0.1912386417388916, 0.8428819179534912, -1.6424486637115479, -1.727340817451477, -1.7771199941635132, -0.6043895483016968, 0.7587814331054688, -0.17079515755176544, -0.1939353495836258, -0.6763177514076233, 1.2736446857452393, 0.370604544878006, 0.5781714916229248, 0.6836459636688232, -0.0030619525350630283, -0.8218276500701904, -0.4275917410850525, 0.6368863582611084, 1.1360305547714233, -0.19551534950733185, 0.8642268776893616, 0.08563870191574097, 0.10800395905971527, 0.10323724150657654, 0.504618763923645, 0.9901986718177795, -0.4246761202812195, -1.6056379079818726, 0.9186162948608398, 0.36694926023483276, 1.4907406568527222, 0.0014910631580278277, 0.8109435439109802, 0.14030316472053528, 1.0779640674591064, -0.8873509168624878, -0.051962751895189285, -1.4733059406280518, 0.7934986352920532, 1.0819538831710815, 0.5088120698928833, 2.9240827560424805, 0.014439528807997704, 0.3877592980861664, 2.7028331756591797, -0.9583770036697388, -1.172878384590149, -0.298394113779068, 2.014948606491089, -1.0188512802124023, 0.4521923363208771, 0.24721071124076843, 1.5500189065933228, 0.4277571141719818, -0.5818508863449097, 1.0085840225219727, 0.37387505173683167, 1.346240520477295, -1.1288944482803345, 1.1961021423339844, -1.0620938539505005, 0.2625609040260315, 0.5630489587783813, -0.6913024187088013, 0.2704174816608429, -1.6925196647644043, 0.9459323287010193, 0.06382133811712265, 0.4024147391319275, -0.9759358167648315, -0.8712055683135986, 0.4825679063796997, 1.7134381532669067, -1.0977998971939087, 0.41762450337409973, 0.7344948649406433, -1.4013851881027222, -1.3034125566482544, 0.15412230789661407, 1.3000996112823486, -0.7500988841056824, 0.9351092576980591, 0.40614762902259827, 0.13925647735595703, -1.4344432353973389, 0.2892697751522064, 1.3759349584579468, -1.0854095220565796, -0.7513359189033508, 0.017611896619200706, 0.1390777826309204, 1.6804789304733276, -1.407585859298706, 1.2703531980514526, 0.6159661412239075, -1.1722655296325684, 1.2725119590759277, 0.9482653141021729, -0.35386013984680176, 0.23360960185527802, -0.18140392005443573, -0.13189783692359924, -0.6969999670982361, 0.5743535757064819, -0.9587932825088501, -0.16400717198848724, -0.21939387917518616, -0.24159495532512665, 0.5805807709693909, 0.47031381726264954, -0.9402652978897095, 0.42430245876312256, 0.40601128339767456, -0.7975389361381531, -0.15528422594070435, 0.7197044491767883, -2.1239984035491943, -0.2478782683610916, -1.7706642150878906, 0.5793520212173462, 0.19822102785110474, -0.2100990116596222, 0.2473004311323166, -1.6808308362960815, 0.21880078315734863, 1.5231289863586426, 0.9841468930244446, 1.0723735094070435, -1.65402352809906, -0.6752877235412598, 0.7352504134178162, -0.16140921413898468, 3.973130702972412, 2.0713083744049072, 0.4592971205711365, -0.06560052186250687, 0.3601582646369934, -0.018789734691381454, 0.2768078148365021, 1.4013490676879883, 0.2694288492202759, -1.3548789024353027, -1.2543964385986328, -0.9161393642425537, 0.812874436378479, -0.599768340587616, -0.03466354310512543, 0.8801903128623962, 1.679383397102356, 0.4928784668445587, 0.8440741896629333, -0.48538529872894287, -0.3809155821800232, 1.343178153038025, -1.8070108890533447, 0.0012372490018606186, -2.159224271774292, -2.0619101524353027, 0.2401736080646515, 0.6248793005943298, -0.1250138282775879, -0.9598807692527771, -1.076130747795105, -0.4145270884037018, -1.0660970211029053, 0.9467713236808777, -0.05574965849518776, 0.9364529252052307, -1.794075846672058, -0.23169587552547455, 0.9008962512016296, -0.46394413709640503, -1.8890323638916016, -0.8355827927589417, 1.3032084703445435, -0.4855315685272217, -0.46853107213974, 0.9844725728034973, -1.313519835472107, -0.7795860767364502, 0.9317667484283447, 0.18791157007217407, -0.7079295516014099, 0.2630946636199951, 2.0717549324035645, -1.2292499542236328, -1.0914814472198486, -0.10304447263479233, 1.2444546222686768, 0.53615802526474, 0.44911015033721924, -0.4471667408943176, -0.40480297803878784, -0.6438601016998291, 1.2070103883743286, -1.4338220357894897, -0.8496723175048828, -0.3741828501224518, -0.3520301282405853, -1.2779271602630615, 0.47501271963119507, -1.0278114080429077, -0.8944427371025085, -0.5335531234741211, 0.6940727829933167, -0.5421580076217651, -0.17583979666233063, -0.14949093759059906, 1.3902307748794556, -1.1205135583877563, 1.3648537397384644, 0.5977070331573486, 2.413644313812256, 0.09385242313146591, 0.5579168796539307, -1.045312762260437, 0.23245228826999664, 0.09353772550821304, -1.0927512645721436, 2.583885669708252, 1.5238535404205322, -0.6019523739814758, 0.01880146935582161, -0.2352564036846161, 1.8314162492752075, -1.8039944171905518, 0.19120025634765625, 0.2317633479833603, 1.1061135530471802, -1.3699114322662354, -0.09111173450946808, 2.3750808238983154, -0.6515104174613953, -0.13139082491397858, -2.2211172580718994, 1.1388763189315796, -0.3108008801937103, 1.1105034351348877, 0.860612690448761, -0.07804278284311295, 1.3446893692016602, 1.0149693489074707, 0.007880390621721745, 0.4342069923877716, 0.9677577018737793, -0.528677225112915, -1.4018502235412598, -1.8620595932006836, 0.05942017212510109, 1.3350141048431396, 0.2905704379081726, -1.6341233253479004, 0.180026575922966, -0.6079538464546204, -0.4850928485393524, 0.576440691947937, 0.3382173180580139, 0.1335674524307251, -0.0022248155437409878, 1.6004022359848022, -1.1322085857391357, -0.6272462606430054, -0.4253164827823639, 1.483472228050232, 0.464690238237381, -1.0291297435760498, 1.313285231590271, -1.2776168584823608, -0.8111183047294617, -0.8815966844558716, 1.3453058004379272, -0.7000923156738281, -0.9966093301773071, 0.8034763932228088, 0.9133965969085693, 0.0597199946641922, 0.7828341126441956, -0.2387007474899292, -0.6787756681442261, 0.5556304454803467, -0.039912834763526917, -0.5704278349876404, 0.20602867007255554, 0.5673644542694092, -0.3808169662952423, -0.6022272706031799, -0.01956806518137455, -1.006671667098999, 0.4996076226234436, -0.12239760160446167, 0.4068043529987335, 1.3784149885177612, 0.870305061340332, 0.5397403240203857, 0.4106108546257019, 1.1437053680419922, 0.4571790397167206, -2.053682327270508, -0.5265347957611084, 1.4616265296936035, 0.06493057310581207, 0.08329635113477707, 2.389681339263916, 0.06988698244094849, 0.5015021562576294, -1.6172301769256592, 0.6346889138221741, 0.016147015616297722, 0.4659807085990906, 1.0443634986877441, -0.21848124265670776, -2.2710468769073486, 0.9110128879547119, 1.8758440017700195, 0.28970444202423096, -0.3629753291606903, -0.08254962414503098, -0.08773453533649445, -0.6784026026725769, -0.8789189457893372, -2.114231586456299, 0.40594443678855896, 1.02646005153656, 0.4576050341129303, -0.14304332435131073, -0.13166093826293945, 1.1144201755523682, 0.6696879267692566, 1.4303261041641235, 0.8244144320487976, -0.14627529680728912, 0.5547441244125366, -0.8206004500389099, 0.1776663213968277, 0.9611853361129761, -0.32692185044288635, -0.528906524181366, 0.07807357609272003, 0.9597043395042419, -1.1267787218093872, 0.02987692877650261, 0.1294030398130417, -0.2953662872314453, 1.2476977109909058, -0.08767851442098618, -1.183428168296814, 1.3784269094467163, -0.21795262396335602, -0.22281238436698914, -0.8018786311149597, -2.3134467601776123, 0.8807886242866516, -0.4012942910194397, 0.03475375100970268, 0.7492314577102661, -0.21490712463855743, 1.7072880268096924, 0.29424014687538147, -1.7628734111785889, -1.3333649635314941, 0.7823445796966553, -0.3909643292427063, -0.5750860571861267, -1.0546398162841797, 2.0664141178131104, -0.31614869832992554, -0.26838722825050354, -1.2608386278152466, 0.4252152740955353, 0.5241743326187134, 0.2575918138027191, -0.10675200074911118, -0.8894598484039307, 0.8255953192710876, -0.7567106485366821, -0.31151777505874634, -0.6815023422241211, -0.9781150817871094, -0.09842778742313385, 0.2010389119386673, 0.5825360417366028, -0.37804114818573, 0.11364541947841644, -0.838754415512085, 0.738381028175354, -1.0028820037841797, -0.4147741198539734, 0.11347010731697083, -0.2379358857870102, -0.1087956428527832, 0.044013746082782745, 1.9434894323349, -1.916462779045105, 1.4213236570358276, 0.9970130324363708, 0.8831440806388855, -0.9892739653587341, -1.3639169931411743, -1.1570802927017212, 1.4430902004241943, -2.08701229095459, 0.4502611756324768, -2.064305305480957, 0.06259000301361084, -0.5260936617851257, -1.3341064453125, -1.0480247735977173, 0.013627482578158379, 0.12103349715471268, -1.7289634943008423, 0.067665234208107, -0.6624762415885925, -1.642181396484375, 1.2944903373718262, 0.9390485286712646, 0.5788509249687195, 1.4953933954238892, 1.119626760482788, -0.5510430335998535, 0.18479308485984802, -0.5785800218582153, 0.03519095852971077, -0.3725072145462036, 1.5153083801269531, -1.676707148551941, -1.561568021774292, -0.34668025374412537, 0.7371768355369568, 0.4484856426715851, -1.2808791399002075, -0.8690611720085144, -0.9712897539138794, -0.17865076661109924, 0.11235781759023666, -0.07928808778524399, -0.5116704106330872, -1.0094839334487915, -0.06860091537237167, 0.08474578708410263, -0.5337271094322205, -1.4380401372909546, 0.6241364479064941, 1.2915256023406982, 1.6012874841690063, -0.38853418827056885, -0.454838365316391, 0.010031625628471375, 0.9611400961875916, -0.8477722406387329, 0.9378668069839478, 0.20965208113193512, 0.6525806188583374, 1.226934552192688, -0.6413154006004333, 0.3099003732204437, 0.006370898336172104, 0.4524376690387726, 0.662617564201355, -0.2278115600347519, 1.251189112663269, -0.2665432095527649, -0.7105323672294617, 1.080270528793335, -0.9732787013053894, -1.1405820846557617, -0.654001772403717, 0.7700235247612, -0.6887036561965942, 2.4954943656921387, -0.19763366878032684, -0.7478445768356323, -1.0175106525421143, 2.281679153442383, 1.0707502365112305, -0.803196370601654, 1.603938341140747, 0.637190580368042, -0.6330971717834473, -0.5462028384208679, -0.16554149985313416, -0.5178367495536804, -1.3276517391204834, -1.1536061763763428, -1.032355546951294, 2.137132406234741, -0.20995444059371948, -1.5753095149993896, -0.5400380492210388, -0.00784408487379551, 1.2137072086334229, 1.0085573196411133, 0.7737920880317688, -1.561357021331787, 0.2323436439037323, -0.40947356820106506, -0.978277325630188, 0.5905315279960632, -0.17909196019172668, -0.16073395311832428, 0.7285929322242737, -0.2710803151130676, -2.4086825847625732, 0.49283233284950256, 0.34334978461265564, -0.9448931217193604, 0.6324089765548706, -0.6090421080589294, -0.5793600678443909, 0.3838098347187042, 0.46186599135398865, -0.3221634328365326, -0.04291152581572533, -0.9046028256416321, 0.2032899558544159, -0.23702380061149597, -1.051979660987854, 0.6403542757034302, 1.4678932428359985, 0.8154252767562866, -0.29540950059890747, -1.085986852645874, -1.130144476890564, 0.7691471576690674, -1.5889843702316284, 0.51325523853302, 2.564208745956421, -0.5875014662742615, -0.15614208579063416, 0.9222765564918518, -1.3211579322814941, 1.5953184366226196, 0.15001049637794495, -0.1914907991886139, -0.038982994854450226, -0.2877862751483917, 1.4028615951538086, 0.3644547462463379, -0.14378367364406586, 1.133560061454773, -1.18804931640625, -0.18520411849021912, 0.8799283504486084, 1.5063565969467163, -0.30671390891075134, 0.8274315595626831, -0.04221508279442787, 2.0086283683776855, 0.23791900277137756, -0.5783055424690247, 2.110328197479248, 0.9508082270622253, -0.5432755351066589, -0.7624159455299377, -0.37216484546661377, -0.6129658222198486, -0.7105765342712402, -1.2511428594589233, 0.5576269030570984, -0.11367777734994888, 0.47795239090919495, 1.2353734970092773, -0.05231411010026932, 0.3740818500518799, -0.7305811643600464, 0.7011570930480957, 0.8623974919319153, 0.3395823836326599, -0.8526565432548523, -0.18048040568828583, -0.3965371251106262, -0.20079633593559265, -1.6344109773635864, -1.1882578134536743, 0.04064516723155975, -0.9134378433227539, 0.5345677733421326, -0.41378793120384216, 0.6658437252044678, -0.45519956946372986, 0.6378625631332397, -0.7763733267784119, 0.17839457094669342, -0.771674394607544, 0.1821415275335312, 0.5574765801429749, 0.6581400632858276, 0.043171875178813934, -1.4825234413146973, -0.7729963064193726, 0.09943673759698868, 0.5884845852851868, -0.5558453798294067, -0.4469372630119324, -1.5418051481246948, -1.1266098022460938, 0.11175309866666794, 0.9519742131233215, -0.749852180480957, 0.1379723697900772, -2.336138963699341, 0.06672046333551407, 1.443123459815979, -0.9094182848930359, -1.0457855463027954, -0.6386410593986511, 1.0742632150650024, -0.44281500577926636, 0.6405684947967529, -0.2813336253166199, 0.3834201991558075, 0.38149142265319824, 0.09002920240163803, 0.09330076724290848, -0.11959225684404373, 0.8595770001411438, -0.09374570101499557, -0.9251638650894165, -0.9264867305755615, -1.2873461246490479, 0.07446295768022537, -0.5931329727172852, 0.33290016651153564, 1.2955944538116455, -1.280845284461975, 0.3154994249343872, 0.6499828100204468, -0.12470093369483948, -0.12426368147134781, 0.04866554215550423, -0.9764660596847534, -0.6031969785690308, 1.0370070934295654, 0.6235854029655457, -0.8023819923400879, -0.9842574000358582, -1.699665904045105, -0.0033787719439715147, -0.3263138234615326, -0.7187278270721436, 1.114347219467163, -0.04153101146221161, -1.5137197971343994, 1.1567137241363525, -0.42928266525268555, 0.47428441047668457, 0.5367917418479919, -0.2630529999732971, -0.6557006239891052, -0.39980366826057434, 1.371540904045105, 0.19956237077713013, -0.42572933435440063, 1.5316640138626099, 0.6207208633422852, 0.7529959082603455, -0.21905039250850677, -0.4193339943885803, -0.12489920854568481, 0.46922779083251953, 0.33968299627304077, 0.7810055613517761, -0.5875041484832764, 2.1118102073669434, -1.0671972036361694, 1.0102344751358032, 0.14963886141777039, -0.4290059208869934, 0.002549859695136547, 1.9950008392333984, -1.3035988807678223, 0.262147456407547, 0.9650364518165588, -1.4249005317687988, -0.6865352988243103, 0.3946351706981659, 0.4841788113117218, 0.085926353931427, -0.595741331577301, -0.6436492800712585, -2.3443305492401123, -1.2627290487289429, -0.16860300302505493, -0.00039907731115818024, -0.012370667420327663, -0.0969696193933487, -0.6063448786735535, -1.17384934425354, -0.00771024264395237, -0.24337251484394073, -1.9632030725479126, -1.588417410850525, 0.24956592917442322, -1.0182510614395142, 1.1432719230651855, -0.8308165669441223, -0.08697076886892319, 0.7989515662193298, -0.4247228503227234, 0.10570710152387619, 0.6877808570861816, -0.01807435043156147, 0.20007000863552094, -0.5342132449150085, 0.9973179697990417, 0.40556973218917847, 0.593124508857727, -0.10983636230230331, -0.7115090489387512, 1.8425363302230835, -3.035142183303833, -1.6790082454681396, 0.2808227241039276, -0.024134457111358643, 0.40005984902381897, -2.0647659301757812, -1.447899580001831, -1.2134565114974976, -1.8750993013381958, 0.23146584630012512, 0.197478249669075, 0.20153221487998962, 0.72007817029953, 0.500653088092804, 0.6938740015029907, -0.24590866267681122, 0.3802792429924011, -0.8230530023574829, 1.4061816930770874, -0.152601420879364, 2.1868526935577393, -0.005459265783429146, -0.03825530409812927, -1.5296612977981567, 1.7479106187820435, -1.8738144636154175, 0.7724341154098511, -0.5759587287902832, -0.7099562883377075, 1.4340159893035889, -0.5243724584579468, 1.1871088743209839, -0.17361785471439362, 0.6621108651161194, 2.4094436168670654, 0.44435369968414307, 1.4008970260620117, -1.355707049369812, 0.7069265842437744, 1.9346952438354492, 0.056099046021699905, 1.0489542484283447, 0.45236653089523315, 0.12910260260105133, -0.5286595821380615, -1.7738335132598877, -0.4896327555179596, -1.7473137378692627], [-1.6004095077514648, 2.0877864360809326, 1.9371211528778076, 1.2538673877716064, 0.9170182943344116, -0.8496701121330261, 0.6014998555183411, 0.3255912661552429, 1.176561951637268, -0.5422994494438171, -0.8450899720191956, 0.36636608839035034, 0.3213323950767517, -0.9584211707115173, 1.2153832912445068, 0.6444936394691467, 0.8092541098594666, 0.3960447609424591, -1.3821660280227661, -0.7796439528465271, 0.35458114743232727, 0.25958025455474854, 0.2933439612388611, -1.953018069267273, 0.4392915666103363, 0.43430420756340027, -2.032128095626831, 0.610194981098175, 0.5266982913017273, 0.13093575835227966, 1.729778528213501, 0.7484427094459534, -0.5426596403121948, -0.3311230540275574, -2.367044448852539, 0.4186965525150299, 1.7967785596847534, 0.49010998010635376, -0.19666916131973267, 0.3587602972984314, 1.468590259552002, 0.10998716950416565, 0.4701985716819763, -0.39464277029037476, -0.26386740803718567, 0.9383077025413513, -0.6897352337837219, -0.5367448329925537, -0.9672610759735107, -1.064601182937622, -0.8501949310302734, 0.16879741847515106, 0.3896121680736542, -0.8211014866828918, -0.8816187381744385, 1.0005048513412476, -1.3099021911621094, -0.599986732006073, 0.05215032771229744, 1.102953314781189, 0.6997270584106445, 0.5861661434173584, 1.3046692609786987, -0.5328481793403625, -0.49150845408439636, 1.147998332977295, 0.2798856794834137, -0.6352719664573669, 0.4943506419658661, -0.608487069606781, 1.0670613050460815, -0.5110183358192444, -0.04524383321404457, -0.2608608305454254, -0.9327457547187805, -0.8010361790657043, -0.2430916726589203, 1.381391167640686, -0.3190082907676697, -0.9285942912101746, -0.5605155229568481, 1.1323899030685425, 0.873672604560852, 0.5743042230606079, 0.28889474272727966, -0.5629599690437317, 0.32575586438179016, -0.13222531974315643, 0.36825841665267944, -0.6614459753036499, 1.0315110683441162, -0.027172356843948364, -1.0194131135940552, 0.7078282237052917, -1.5625239610671997, -0.7120077610015869, 2.113267660140991, -0.5078213810920715, -1.9305329322814941, -0.4435221254825592, -0.12734398245811462, 0.24509583413600922, -0.44733619689941406, 0.22082869708538055, 0.14918076992034912, -0.10781426727771759, 1.9770597219467163, -0.6830734610557556, 1.219417691230774, 0.5536043047904968, -0.7235052585601807, 0.9524040818214417, 2.02026629447937, -0.402385413646698, -1.1621239185333252, 0.3812234401702881, -1.2201488018035889, -0.7003713846206665, 0.6781949996948242, 0.6911004185676575, -0.12980563938617706, 0.4387136697769165, 0.08301977068185806, 0.9135963320732117, -0.7155910134315491, -1.4148035049438477, -0.7287845015525818, 0.4780408442020416, -1.1356818675994873, -0.1640787273645401, 2.0481834411621094, -0.4335367977619171, 0.057007212191820145, 0.4795960783958435, 0.9843964576721191, -0.7176553010940552, -0.2923946678638458, -1.45280921459198, 1.0651313066482544, 1.8584340810775757, -0.1418788582086563, 1.9139734506607056, -0.35882776975631714, 0.9437171220779419, 0.24160417914390564, 0.15962152183055878, -1.2239654064178467, -0.04581805318593979, 0.797015368938446, -0.9704303741455078, 1.6247917413711548, 1.7225013971328735, 0.3881493806838989, -1.4502414464950562, -0.2601710855960846, 1.3256844282150269, -0.810447633266449, 0.6314773559570312, 0.5253501534461975, -0.8370006084442139, -0.32895904779434204, -0.2518686056137085, 1.6883503198623657, 0.737384021282196, -0.5117350816726685, 0.9719424247741699, 0.6432073712348938, 0.2909962236881256, -0.26407405734062195, 0.7068383693695068, 0.38675418496131897, 0.22064508497714996, -0.1481013298034668, -0.42701613903045654, -0.870633602142334, -1.6525743007659912, -0.6014153361320496, 0.38590487837791443, 2.2415645122528076, 0.9832503199577332, 0.11647121608257294, 1.0387178659439087, -0.20710629224777222, -0.5808053016662598, 0.7443215250968933, -1.8384099006652832, -0.4479624032974243, -0.7865463495254517, -0.527766764163971, -0.8729162216186523, -0.953307032585144, 0.5666826963424683, 1.6013660430908203, 1.1816046237945557, -1.1611478328704834, -0.4942655861377716, 1.1884353160858154, 0.49149659276008606, 0.360830694437027, 1.0603550672531128, -0.7155451774597168, -0.5460577011108398, 1.071455478668213, 0.6211915016174316, -0.8771555423736572, -1.833057165145874, 1.9722293615341187, 1.8768151998519897, 1.8768491744995117, -1.7847977876663208, -0.27790096402168274, 1.6459068059921265, 0.49566957354545593, 2.758399248123169, -0.45512261986732483, 0.1836766004562378, -0.27620774507522583, -0.8781527876853943, 0.8323848247528076, 0.34356287121772766, -2.1779298782348633, -0.8313054442405701, -0.9618157744407654, 1.218876838684082, 0.1605151742696762, -1.8263158798217773, -1.264470100402832, -1.266308307647705, 0.9513460993766785, 0.44752129912376404, 0.31707602739334106, -0.469680517911911, 0.15730036795139313, 1.2313152551651, -0.4722602069377899, -0.22287200391292572, -0.3101794123649597, 0.5475489497184753, 1.5250113010406494, -0.12728358805179596, -0.24071048200130463, 0.11548881232738495, -0.5135270953178406, -0.1942216157913208, -0.6182382702827454, -0.04557989910244942, 0.17711637914180756, -0.9244783520698547, -1.124974012374878, -0.6435332298278809, 1.3779616355895996, 0.23588068783283234, -1.757095217704773, 0.24503768980503082, 0.24585801362991333, 0.4848771393299103, -1.5167752504348755, -0.9412230253219604, -0.3509474992752075, 0.6172239780426025, -0.05164427310228348, -0.304867684841156, -0.1880149245262146, -1.0526401996612549, -0.2116033434867859, -2.1269683837890625, -0.660978376865387, -0.6704968214035034, 0.1613161712884903, 0.017046578228473663, -0.8028150200843811, -0.42628738284111023, -2.3772666454315186, 0.1254846751689911, 0.8183502554893494, 1.015510082244873, -0.0997542142868042, -0.15807034075260162, -0.09759820997714996, -0.01782531850039959, 0.4317852556705475, -0.23631615936756134, 0.3350185453891754, -0.6789265871047974, 1.1731446981430054, -0.6711965799331665, 0.04146497696638107, 0.1617933064699173, -0.8787951469421387, 0.9829843640327454, -0.6114774346351624, 1.6631462574005127, 1.2394895553588867, 1.0524410009384155, 2.5682218074798584, 1.1670738458633423, 0.08924296498298645, 0.561172604560852, -0.6070722341537476, 2.0771701335906982, -0.9380162358283997, 0.7706838250160217, -0.013735320419073105, 0.34401586651802063, 0.9923356175422668, 0.34612587094306946, -1.345784306526184, 0.04320627450942993, -0.31268924474716187, 0.11653541028499603, -0.04189502075314522, -0.3528802990913391, -0.871833860874176, -1.1768136024475098, -1.168300986289978, 1.235072374343872, -0.7755545377731323, 0.41393354535102844, -0.09703974425792694, -0.15654540061950684, 0.40558403730392456, -1.4386416673660278, -0.9086849689483643, -1.71707022190094, 1.642913818359375, 0.6887199878692627, 0.9389487504959106, -0.8223539590835571, -1.5114623308181763, -0.0344821996986866, -0.8901786208152771, -2.4714462757110596, 0.5062955021858215, -1.3019095659255981, -1.346325159072876, -0.29666683077812195, 0.6621304154396057, -1.0706076622009277, 0.8133223652839661, -1.0820058584213257, 1.5545531511306763, 0.30241718888282776, -0.9626551270484924, -0.4378356635570526, 1.9660660028457642, -0.5864349603652954, 1.6536295413970947, 1.1725300550460815, -1.2129576206207275, 0.44433239102363586, -0.3196846842765808, 0.9790069460868835, 0.18246343731880188, 0.018304869532585144, 0.05436597391963005, 1.3886041641235352, 0.253887414932251, 1.2218679189682007, -0.8407813906669617, -1.2509393692016602, -0.8671510219573975, 0.0981753021478653, 0.17081822454929352, 0.36093249917030334, -1.7469879388809204, -1.5778746604919434, -0.3765724301338196, 0.12924104928970337, -0.19311581552028656, -0.02816464938223362, -0.561755359172821, 0.10301078110933304, 0.5356661677360535, -2.3115482330322266, 0.9067058563232422, -0.9322166442871094, 0.2114206850528717, 0.9378073811531067, -0.5246753096580505, -0.45838674902915955, 1.9873706102371216, -0.31037408113479614, 0.12567418813705444, 0.47542813420295715, -0.22600756585597992, -0.3992752134799957, -1.0825421810150146, 0.7469019889831543, 1.185821533203125, -0.7244395017623901, -0.40262138843536377, 0.494498074054718, -0.7857283353805542, -0.3265061676502228, -0.7101603150367737, 0.08677102625370026, -0.4973934590816498, -0.723235011100769, 0.6669411659240723, -0.5805963277816772, 0.31305384635925293, 1.1835830211639404, -0.6396355032920837, -0.8629213571548462, -1.2044247388839722, -1.5673829317092896, 0.7895996570587158, 1.013253927230835, -0.6121866106987, -1.5369728803634644, -0.4014606475830078, -0.09065281599760056, 0.18164463341236115, -1.4338127374649048, -1.437654733657837, -1.7408051490783691, 2.6367833614349365, 0.36099234223365784, 1.0193763971328735, 0.11080843955278397, 0.11479701101779938, -0.36920103430747986, -0.897221028804779, 0.8705109357833862, -0.5758000016212463, -0.5966601371765137, 0.5706878304481506, -0.03645910322666168, -0.018580907955765724, -0.2757340669631958, -0.026921862736344337, -0.6538458466529846, 0.12295256555080414, -1.0362015962600708, -1.071069598197937, 0.663788378238678, -0.9799850583076477, 1.7297935485839844, -0.8551161289215088, 0.650880753993988, -1.1404767036437988, 0.9713037014007568, 1.2757213115692139, -0.3012292981147766, 0.7206770181655884, 1.0960628986358643, -1.1474201679229736, 0.1610797494649887, 1.6161397695541382, 0.7751601338386536, 0.20440803468227386, 1.037863850593567, -0.16315077245235443, -1.2799813747406006, 0.8850016593933105, -0.9359558820724487, -1.1282906532287598, -0.7667495012283325, 1.560977816581726, 2.071794033050537, 0.6779511570930481, -1.6601533889770508, -1.3930543661117554, 0.4569105803966522, 0.9299677610397339, -0.9270654320716858, -0.6804420351982117, 1.746202826499939, 1.295784831047058, -0.07480024546384811, -0.15491534769535065, -0.14647389948368073, 1.105047345161438, 0.25999513268470764, 0.8812565803527832, -1.9090440273284912, -1.2957485914230347, -0.028945816680788994, -0.5174832344055176, -0.2183203399181366, -0.6577260494232178, 0.031720444560050964, -0.686161458492279, 0.4585702121257782, -0.012357856146991253, 0.5871328711509705, -0.6673828363418579, 0.6110795736312866, 2.1167595386505127, -0.3678969740867615, -0.8499121069908142, 1.3247202634811401, 0.9384718537330627, 1.0450037717819214, 0.9357132315635681, 0.7322307229042053, -0.7173088192939758, -0.8233361840248108, 1.1545727252960205, -1.064034342765808, -0.4812145233154297, 0.5304681658744812, -0.019665006548166275, 0.1363941878080368, -2.1929917335510254, -0.6753142476081848, -0.9198020696640015, -1.2906709909439087, 0.3143428862094879, -0.10615628212690353, -0.9505611658096313, 0.6560459136962891, 0.43037423491477966, -0.3659835457801819, 1.2625268697738647, -0.5499900579452515, 1.1955417394638062, -0.47516095638275146, -1.4686938524246216, 1.3561478853225708, -0.8566136956214905, 0.5152244567871094, -0.37857383489608765, 0.6757293939590454, 0.310699999332428, 0.165146142244339, -1.7777678966522217, 0.5296844244003296, -0.7135970592498779, 1.106161117553711, 0.99522465467453, -0.5966607928276062, 0.003249026834964752, 1.5992047786712646, 0.015912184491753578, -1.0034490823745728, 0.027525579556822777, -1.3438209295272827, -1.2260874509811401, 0.8756791353225708, -1.1250790357589722, 0.43652868270874023, 0.06209361553192139, 1.305765986442566, 0.7983585596084595, 0.5417425036430359, 0.06673989444971085, -0.6400442719459534, 0.06887262314558029, -0.09972821176052094, 0.5018949508666992, 0.08181967586278915, -0.16982205212116241, -1.3309017419815063, -1.6799873113632202, 1.6539032459259033, -1.2770583629608154, -1.4060732126235962, 0.15295949578285217, 0.18199323117733002, -0.5724651217460632, -0.6403736472129822, 2.5993411540985107, -0.4916013181209564, -0.9474777579307556, -0.26394104957580566, -1.042603611946106, -1.145590901374817, 0.3199031352996826, 0.4551082253456116, -0.35828500986099243, -0.6381220817565918, 0.16104258596897125, 1.040541410446167, 0.2870083749294281, -0.628301739692688, -0.05855837091803551, 2.4720897674560547, -1.6533931493759155, 0.7911547422409058, 1.1332491636276245, 0.4850773513317108, -0.34309059381484985, 0.7786069512367249, 0.6629752516746521, -0.015339147299528122, -1.453384518623352, 1.2604247331619263, 0.5104835629463196, 0.7675901651382446, 0.4692576825618744, -1.0776340961456299, 2.06689715385437, -0.24918034672737122, -0.9321452975273132, 0.8465710282325745, -0.23967397212982178, 1.0690717697143555, 0.963230550289154, 0.545015275478363, 1.0193367004394531, 0.10802871733903885, -0.039236702024936676, -0.5428948998451233, 0.29693442583084106, 1.3821139335632324, -0.29187947511672974, -0.03550908342003822, 0.7155343890190125, 0.9382514953613281, -0.7251951098442078, 0.20413509011268616, 0.9852203130722046, 0.5588428378105164, -0.5293169021606445, 0.3731389045715332, -1.3712652921676636, 0.09643622487783432, 0.06624924391508102, -0.39709416031837463, 0.408161997795105, -2.0666821002960205, -0.26057958602905273, 0.4879128932952881, -0.033905863761901855, 0.20329166948795319, 1.1553467512130737, 1.3063938617706299, -0.7773965001106262, -0.935492217540741, 0.18249626457691193, -0.23405581712722778, -1.5339231491088867, 0.764655351638794, -1.8186787366867065, 3.4624459743499756, 0.08729714155197144, 1.934889316558838, 0.6472104787826538, 0.4203733503818512, -0.4356963336467743, -0.7559973001480103, 0.9036176800727844, -1.0879380702972412, -0.4871675670146942, -0.018047839403152466, 1.9591765403747559, 1.0662572383880615, 0.26745668053627014, 0.24799855053424835, 0.15799854695796967, 1.6467934846878052, -0.3877926766872406, -0.9089145660400391, 0.10187078267335892, -1.598659634590149, 1.5088332891464233, -0.6622837781906128, 0.7170812487602234, -1.244760274887085, -0.5337194800376892, -0.14603200554847717, -0.5403587222099304, 0.41393521428108215, -1.0529439449310303, 0.4083879888057709, 1.063295602798462, -0.3581661283969879, -1.7516775131225586, -2.0449109077453613, 1.138222098350525, 0.24299229681491852, -0.574038028717041, 0.0417599081993103, -0.749504804611206, 1.6327697038650513, -3.268211603164673, 0.15836773812770844, -1.169321894645691, 0.1330096274614334, -0.5918079018592834, 1.4070510864257812, -0.2801242172718048, -0.3996160328388214, 2.0955634117126465, 0.26950132846832275, 0.5692493915557861, -0.1300216168165207, 0.23597955703735352, 0.21498534083366394, -0.013181782327592373, 0.45149165391921997, -0.506723940372467, -1.7368277311325073, 1.2251269817352295, 0.6931535601615906, -0.2428002655506134, -0.864355742931366, 0.9502188563346863, -0.7646106481552124, 0.051375653594732285, 2.6516880989074707, -1.461443543434143, 0.38476455211639404, -0.20143035054206848, 0.15635839104652405, 0.929553210735321, -0.8487342000007629, -1.9562281370162964, -1.6624047756195068, 0.5317639708518982, -0.12015094608068466, 0.5751561522483826, 1.1135804653167725, -0.33203333616256714, -2.9480648040771484, -0.5498790740966797, -0.4501316547393799, -0.05176801607012749, 1.3346483707427979, 1.2079180479049683, -1.4502817392349243, 0.024872228503227234, -0.5836607217788696, 0.46387261152267456, -1.2186692953109741, 0.4333758056163788, -0.11520693451166153, -0.5863537788391113, -0.6751062870025635, 1.1035475730895996, -1.0082916021347046, 0.3414911925792694, 0.2229078859090805, -0.753984808921814, -0.1145571693778038, 0.8622824549674988, -2.421204090118408, -0.0637604221701622, -0.8972779512405396, 0.34381043910980225, 0.9922482371330261, 0.600614607334137, -0.9574483633041382, -0.05786386504769325, 0.7501644492149353, -0.4361530542373657, 1.3386127948760986, -0.9299731254577637, -0.6679866909980774, 0.6428037285804749, -0.33412137627601624, 0.6371073126792908, -2.140859365463257, -0.0047856527380645275, 0.16551458835601807, 1.3646565675735474, 0.8477910757064819, -0.8395613431930542, 0.9209187030792236, 0.747464120388031, -2.016063690185547, -0.1650257408618927, 1.2835062742233276, -1.6098202466964722, -0.30092930793762207, 0.4419572651386261, 0.10188962519168854, 0.5588817000389099, -1.2879745960235596, -1.4601633548736572, -0.7572954893112183, -1.0567264556884766, -0.12069227546453476, -2.1123342514038086, 0.30253252387046814, 1.4262018203735352, -0.05127200111746788, 2.497224807739258, -1.7150720357894897, 0.4111478328704834, 0.8247243762016296, -1.1694233417510986, 1.724595546722412, 1.4668233394622803, -0.48255038261413574, 1.5605138540267944, -0.7486889958381653, -0.008899861946702003, 1.4629212617874146, -1.4146517515182495, -1.694525122642517, 0.3882955014705658, 0.022857628762722015, -0.09144137799739838, 0.6733373999595642, 0.520914614200592, 0.6979143023490906, 0.7530682682991028, -0.9186275601387024, 0.5134404301643372, -0.16140872240066528, 1.1298807859420776, -0.15055038034915924, -2.424368143081665, 0.030384350568056107, -0.9301555156707764, -1.0087366104125977, -0.7115621566772461, 0.4260674715042114, 0.9947041273117065, 1.7374347448349, -1.6109955310821533, 0.04291301220655441, 0.6273127198219299, 0.9419999122619629, 0.1840352714061737, -0.21302810311317444, 0.7449720501899719, -0.27171963453292847, 0.20987965166568756, -2.0606279373168945, -0.43405333161354065, 0.42598751187324524, 0.13056820631027222, 0.6336846947669983, -0.1340734362602234, -0.47982844710350037, 1.3001021146774292, 0.43886199593544006, 0.43981051445007324, 0.5117095112800598, 0.008202123455703259, 0.8855867981910706, -1.3670183420181274, 0.07583463191986084, 0.8001179695129395, 0.12922392785549164, 0.8605692982673645, 0.0598042756319046, -0.5988920331001282, 0.594336986541748, -0.6777081489562988, 1.512233853340149, -1.9171615839004517, 0.4880530834197998, 0.0395750068128109, 0.30676552653312683, -0.11310812830924988, 1.7476943731307983, -1.4288216829299927, -0.9515252113342285, 1.1093250513076782, -0.2631034851074219, -0.5313096046447754, -0.2561365067958832, -0.005488508380949497, -0.15665264427661896, -0.6171600818634033, 0.40433040261268616, -0.22081059217453003, 0.42847174406051636, -0.9589921236038208, -1.173653244972229, -0.3031231164932251, 0.5074400901794434, -0.9661105871200562, 1.9425657987594604, -0.33449122309684753, -0.8201624751091003, -0.1531737744808197, 0.40415677428245544, -0.5317115187644958, -0.6608225703239441, -2.0308282375335693, 0.19763721525669098, 0.041780002415180206, 1.1435184478759766, 1.6741721630096436, -1.7219380140304565, 0.9861554503440857, -0.9609246850013733, 1.5595732927322388, -2.685192584991455, -1.4131687879562378, 1.7963261604309082, 0.474796861410141, -0.28209608793258667, -1.317341923713684, -1.9698026180267334, -1.06519615650177, -0.1437053084373474, 1.0638712644577026, 1.899649977684021, -0.6850060224533081, -1.7416812181472778, -1.04163658618927, 0.49420055747032166, -0.20923933386802673, 0.9923417568206787, 0.13138672709465027, -0.2313762754201889, 0.7132565975189209, -0.14353542029857635, -1.155001163482666, -0.4337611794471741, -1.2042380571365356, 0.005268997512757778, -0.12876419723033905, -0.2556307911872864, 0.7673009037971497, -0.3686359226703644, 1.3360544443130493, 1.256197214126587, 1.5415483713150024, 0.41315755248069763, -0.022126754745841026, -1.7757484912872314, 1.6073979139328003, 1.104013442993164, -0.8857793211936951, 0.21916349232196808, 0.7107628583908081, -0.01489389780908823, 2.466801881790161, -0.29566690325737, 0.36277931928634644, 0.9588367938995361, 0.5068788528442383, -2.6188747882843018, -0.5093576312065125, 0.9427223205566406, -0.3568772077560425, -1.4312472343444824, -0.4448404312133789, -0.7898322939872742, 1.9511176347732544, 1.7147578001022339, 0.013060336001217365, -0.8192317485809326, 1.3489447832107544, 0.06869383901357651, 0.573986828327179, 0.8195456266403198, -1.3294240236282349, 0.24962913990020752, 0.2612907886505127, -0.6112021207809448, 0.19818024337291718, -0.1258212774991989, -0.8546271920204163, 0.18834081292152405, -1.0845240354537964, -0.233052060008049, -1.5718231201171875, 1.2946449518203735, -0.9238258004188538, 1.7991162538528442, -0.02663434110581875, 0.34190112352371216, 0.558129072189331, 0.06314947456121445, -0.8795596957206726, 0.39292943477630615, 0.8532025814056396, -0.5075046420097351, -0.1589130163192749, -1.1753652095794678, -1.4746507406234741, 0.4423089921474457, 0.5345640778541565, 0.6144028306007385, 2.6786837577819824, -1.167065978050232, 0.5206754207611084, 2.076488971710205, -1.6219617128372192, -1.076246976852417, -1.1564522981643677, -0.10289227962493896, 0.6382626295089722, 0.1688644289970398, -0.6433233618736267, 0.22968533635139465, 0.3766095042228699, -0.18798425793647766, -1.059916615486145, -1.094834566116333, -0.7848619222640991, 0.863629937171936, 1.1662743091583252, 0.579326868057251, 0.48912909626960754, 1.243382215499878, 0.6237504482269287, -1.229966163635254, 0.391258180141449, 0.15047100186347961, 0.35145819187164307, -1.4757474660873413, -0.5308163166046143, 0.766370952129364, 0.7841579914093018, 0.8044607043266296, 0.05215820297598839, 0.07216452807188034, 0.6062759160995483, -0.7376296520233154, 0.0191364586353302, 0.8636379241943359, 1.0429853200912476, 0.20871490240097046, 1.679303526878357, 1.715868353843689, 0.9363374710083008, 1.3486727476119995, 0.020967477932572365, 0.8172190189361572, 0.18489117920398712, 2.0300235748291016, -0.43845024704933167, -1.4705287218093872, -1.2254493236541748, 0.3231005370616913], [0.9708510041236877, 0.2654116451740265, -0.16033440828323364, -0.4799683094024658, 0.29680123925209045, -0.20581167936325073, -0.6282593011856079, -0.2804352641105652, 1.4710006713867188, 0.2602746784687042, -1.0342211723327637, -0.969005286693573, -1.2277199029922485, -0.5279186964035034, 0.4786928594112396, -0.4997601807117462, 1.9379754066467285, 0.06774915009737015, 0.1618579775094986, 0.5934653282165527, -1.5982842445373535, 0.7731809020042419, 2.674093008041382, 0.7843349575996399, -1.166945457458496, -0.2711062431335449, 1.3948947191238403, 0.8568575978279114, -0.5063261389732361, -1.1520875692367554, 0.4759226441383362, 1.6887972354888916, -0.35617178678512573, 0.4786932170391083, -0.5546677708625793, 0.022310640662908554, 0.4187636375427246, -1.0567362308502197, 0.49615100026130676, 0.7783240079879761, 1.168078899383545, -1.8583621978759766, 0.9881492853164673, -0.05073787644505501, 0.6266617774963379, -0.7105282545089722, 0.08147351443767548, -0.7183273434638977, -1.653083324432373, 0.2866417169570923, 0.6044058203697205, -1.0558772087097168, -1.5467616319656372, 1.0458191633224487, -0.4553382396697998, 0.8233341574668884, 0.6254038214683533, -2.332679510116577, -0.9167329668998718, 0.8771808743476868, -2.1597511768341064, -1.7367490530014038, 0.4222622513771057, -0.4702893793582916, -0.49882078170776367, -0.8508177995681763, 1.5581037998199463, -1.3706409931182861, 1.585984230041504, -0.3216140568256378, -0.15695221722126007, 0.2692449986934662, 0.0629090741276741, 0.6876338720321655, 0.03442508727312088, 0.1993495374917984, -0.13897432386875153, 1.2401988506317139, -0.9071922302246094, -1.8115055561065674, -0.5025228261947632, -0.6446261405944824, 0.3538189232349396, 0.6733074188232422, 0.466126412153244, 0.44793635606765747, -0.7348461747169495, 0.6325098872184753, -1.663435935974121, 1.2764722108840942, 0.19289274513721466, -0.4309675395488739, -0.32080206274986267, 0.7531083226203918, 1.1927262544631958, 0.1057499423623085, -1.1268479824066162, -1.7625633478164673, 0.6948633790016174, -0.763400673866272, 0.863487184047699, 0.4083835780620575, 1.045021414756775, -0.8508981466293335, -0.35830995440483093, 1.8558725118637085, -1.7657074928283691, -1.2531930208206177, 0.16682367026805878, -0.0714910700917244, -0.3613188862800598, 0.38990700244903564, -0.3876563608646393, 1.1744987964630127, 0.17442375421524048, -0.028483550995588303, -0.018226079642772675, -0.509426474571228, 0.498043954372406, -0.3740362226963043, 0.8353990316390991, -0.6286863684654236, -0.5405586361885071, 1.0167698860168457, -0.22362400591373444, -0.7133291959762573, -0.8450426459312439, 1.0340951681137085, 0.2068764716386795, -0.4926608204841614, -0.5563178062438965, -0.0493631549179554, -1.5389134883880615, -0.4972287714481354, 1.2774475812911987, -1.5147336721420288, -0.6133350133895874, 1.2918107509613037, 0.30016398429870605, -0.41600877046585083, 0.3658476769924164, 0.019950853660702705, 1.3604778051376343, -1.3688268661499023, 0.6054338812828064, -0.3706493675708771, 0.10594942420721054, -0.6142851114273071, -0.09157200157642365, 0.33041703701019287, 0.6792465448379517, -1.1268161535263062, 0.036429960280656815, 2.021838665008545, 0.4196411371231079, -1.4197615385055542, 0.3618326485157013, -0.5853740572929382, -0.15153057873249054, 0.39521437883377075, -0.5412364602088928, 1.7189717292785645, -0.47769707441329956, 0.37232932448387146, 1.2705488204956055, 0.3170643448829651, 1.6396476030349731, -1.9485278129577637, 1.0246537923812866, 0.1391681432723999, -1.1207302808761597, 2.2333409786224365, 1.209944725036621, -0.04573746398091316, -0.8752416968345642, -2.030353546142578, 0.08743412792682648, -0.9375860095024109, -0.7566691040992737, 0.06442304700613022, 0.006549806799739599, -0.20950473845005035, -0.18907596170902252, 0.05107150226831436, 0.7373430132865906, -0.3118009865283966, 0.7944406270980835, -0.6755565404891968, 0.6559101939201355, 0.12922342121601105, 1.476838231086731, -1.0822393894195557, -0.43682971596717834, -1.2339214086532593, -1.6301084756851196, 0.8922852277755737, -0.07091572880744934, -1.3116763830184937, 0.8566288948059082, 0.31997525691986084, 0.6647447943687439, 0.3098386824131012, 0.6920019388198853, 0.2313559651374817, -0.1323283463716507, -0.3846275806427002, 0.12193144112825394, 0.18068762123584747, -0.6706923246383667, 0.5294498205184937, 0.7545202970504761, 0.9592500329017639, 0.2828674614429474, 1.6554487943649292, -1.386332392692566, -0.5241805911064148, 0.11320704966783524, 0.336210697889328, 0.26102668046951294, -0.09991607815027237, 0.5965730547904968, 0.09504186362028122, -0.5949665904045105, -1.1584614515304565, -0.672652006149292, -1.884291410446167, -0.9516569972038269, 0.5015463829040527, 0.37825632095336914, -0.6159745454788208, -0.4807374179363251, 1.1155915260314941, 0.7890660762786865, 0.7366156578063965, -1.7679314613342285, 1.6064680814743042, 0.6090139746665955, 0.8767102360725403, 1.4076677560806274, -0.410543292760849, -0.15658026933670044, 0.2256932556629181, 1.524788737297058, -0.9995851516723633, 1.6210130453109741, -0.2010750025510788, 1.1483104228973389, 2.599881649017334, 1.0033960342407227, -2.1426215171813965, 1.5477195978164673, -0.5819988250732422, -0.14714568853378296, 1.2015199661254883, -0.6998549699783325, -0.032605092972517014, -0.6374601721763611, 0.3337438702583313, -1.4776147603988647, 0.16784758865833282, 0.2570752799510956, 0.47521495819091797, 1.1357170343399048, 2.6894874572753906, 0.5526812672615051, -0.29451003670692444, 0.06186935678124428, 0.6147685050964355, 1.0577338933944702, 1.3746044635772705, -0.6873994469642639, -0.868889331817627, -2.2282261848449707, 1.99882173538208, -0.8090217113494873, -0.04657099395990372, 0.1188230812549591, -0.989877462387085, -0.5960396528244019, 1.1926896572113037, 0.18085582554340363, 0.7071669697761536, 0.0007366101490333676, 0.410540372133255, -0.9544022679328918, 0.10509446263313293, -0.7542950510978699, 0.0543106272816658, 0.5809207558631897, 0.5276418328285217, 3.203554630279541, -0.46095573902130127, -0.43160274624824524, 1.2142307758331299, -0.15479496121406555, -1.0714683532714844, -0.968553364276886, -0.44809722900390625, 0.47487860918045044, -0.8748359084129333, 1.965248942375183, 0.26212236285209656, -0.4567146301269531, -0.6326229572296143, -0.8366824984550476, 0.7244420051574707, 0.8090257048606873, -0.6526442766189575, -0.2650969922542572, 1.282961368560791, -0.5138376951217651, -0.8200181722640991, -0.07061897963285446, 0.5423402190208435, 1.0541331768035889, 0.766697347164154, 0.29839572310447693, -0.12499634921550751, 1.059106707572937, 3.3959386348724365, 0.34048235416412354, 0.7610370516777039, -0.41949453949928284, -0.0319499671459198, -0.6971403360366821, -1.7841697931289673, 0.3589674234390259, -0.5053451061248779, -0.0003332523338031024, 1.0080275535583496, -0.5295188426971436, 0.28866147994995117, -1.0042893886566162, -0.9553026556968689, -0.6908610463142395, -0.15914012491703033, -0.23553186655044556, -0.49472981691360474, 2.1992135047912598, 1.6953696012496948, 0.5433434247970581, 0.7028051018714905, -1.751914620399475, 0.45102518796920776, 0.20847846567630768, 0.3298303484916687, -0.10189823061227798, -0.2915080189704895, 1.8057096004486084, 0.07135267555713654, 0.5464826822280884, 0.46349504590034485, 0.22608987987041473, 0.5406866073608398, 1.4703738689422607, 0.8955159187316895, 1.3101216554641724, -1.599746823310852, 0.4815118908882141, -0.24071593582630157, 0.9058405160903931, 0.26116886734962463, 0.30257898569107056, 0.8161584734916687, -0.5439250469207764, -1.1395372152328491, 1.097896695137024, 0.4488934874534607, 0.16465188562870026, 0.29006227850914, -1.1473205089569092, 0.5585211515426636, -0.03946075215935707, -1.2206310033798218, -1.0207459926605225, -0.6866245865821838, 1.0180059671401978, 0.8219546675682068, -0.09824332594871521, 0.3243443965911865, -0.017711976543068886, 1.8524152040481567, 1.4337716102600098, 0.6341206431388855, 0.9596816897392273, -0.5702158212661743, -0.3474193513393402, 1.4136837720870972, -0.3372015058994293, 0.23298262059688568, 0.12793485820293427, 1.49280846118927, -2.116386651992798, 1.6557799577713013, -0.5593516230583191, -0.7949831485748291, -0.9167468547821045, -0.42178696393966675, -0.05951479822397232, 1.4269254207611084, 0.587378203868866, 0.11348232626914978, 0.13377687335014343, -0.8195751905441284, 0.6458920240402222, -0.979866623878479, -0.37957096099853516, 0.7527914047241211, -1.3297330141067505, 1.5502698421478271, 0.33949968218803406, -1.2530909776687622, -1.027510404586792, -0.9549678564071655, 0.25991275906562805, 1.265623927116394, -1.6705479621887207, -2.0059902667999268, 1.1621572971343994, 2.2070322036743164, -0.852972149848938, -0.7403738498687744, 0.2655889093875885, -0.8019724488258362, -2.0006070137023926, -0.011324070394039154, 0.8433934450149536, 0.41549617052078247, -1.1142915487289429, 0.8631913661956787, 1.2389421463012695, -0.744911253452301, -0.9822591543197632, -0.17689362168312073, -1.1275545358657837, 0.04585036262869835, 0.07128143310546875, -0.48106837272644043, 0.9686366319656372, -0.40533143281936646, 2.121943712234497, 0.5946073532104492, 0.8492740988731384, 1.084989070892334, 0.5558278560638428, -0.7153603434562683, 0.2109324038028717, -0.5795512795448303, -0.2675705552101135, 0.5180492997169495, -0.6907305121421814, 0.8064433932304382, 3.2826058864593506, -0.23480650782585144, 0.24604186415672302, 0.7844411730766296, -0.9498900175094604, -0.942599356174469, -0.9353274703025818, 0.7772204875946045, -1.1148542165756226, -0.8153248429298401, -0.8892592191696167, -0.4444546699523926, 0.5384424328804016, -0.5235132575035095, 1.2492015361785889, 2.1986441612243652, 1.121410846710205, 0.42865556478500366, 2.355361223220825, 0.14050109684467316, 0.258531779050827, 0.17904548346996307, 0.08999254554510117, -2.279057264328003, 0.20633743703365326, 0.6385579705238342, 0.6837508082389832, -0.016465073451399803, -1.108277678489685, -0.11721018701791763, 0.09059475362300873, 0.46566128730773926, -1.6112574338912964, -0.2341768443584442, -1.261065125465393, 0.27166858315467834, 0.9289054870605469, 0.05309827998280525, 0.9541751742362976, -1.0446699857711792, 0.9956261515617371, 0.5182709097862244, -0.6703284978866577, -1.5525553226470947, -1.4796710014343262, 0.30077943205833435, 0.4678477644920349, 1.7540019750595093, 1.0373762845993042, -0.21428810060024261, -0.937627911567688, 1.3489092588424683, 0.26355433464050293, 0.3557814955711365, 1.185649037361145, -1.4971191883087158, -1.0767141580581665, 0.7006435394287109, 1.0316133499145508, 0.22513402998447418, -0.4776975214481354, -0.14182187616825104, -0.7917006611824036, -0.5835118293762207, -0.5657130479812622, 1.065713882446289, -0.4190574884414673, 0.22494062781333923, 0.7732463479042053, 1.456514835357666, 0.38694286346435547, 0.5764207243919373, 1.0911753177642822, 0.6526583433151245, -0.5331476330757141, 1.5161900520324707, 0.14574795961380005, 0.23518557846546173, -0.862436056137085, -0.6474409103393555, -0.9039347171783447, 1.2366565465927124, 0.2215874195098877, -0.16233623027801514, 0.04936382174491882, 1.1293283700942993, 0.08183030784130096, 0.2087046504020691, -0.836355447769165, -0.3449918329715729, 2.220874547958374, 0.4112566411495209, 0.21213895082473755, 0.8095785975456238, -0.2925211787223816, 0.38209739327430725, -0.4706176519393921, 1.5681530237197876, -1.6203522682189941, -0.390739768743515, 2.29691743850708, -0.08678954839706421, -0.8964136838912964, 1.2277443408966064, -0.4076874554157257, 0.8040881752967834, -0.8447446227073669, 1.3960999250411987, 0.6240090131759644, -1.4798232316970825, -0.25521254539489746, -1.5006927251815796, -0.2277386635541916, 0.5960844159126282, -0.8109714388847351, 2.896003246307373, 0.758827269077301, 2.5108957290649414, -1.076381802558899, 1.2618331909179688, -1.6134543418884277, 0.12568630278110504, -0.1325463354587555, 1.0894275903701782, -0.7695159316062927, -0.9741134643554688, -0.0010806974023580551, -0.9641116857528687, -0.25207918882369995, 0.46577024459838867, -0.15438635647296906, 0.13864651322364807, -0.7648799419403076, -0.21377718448638916, -0.9308683276176453, -0.22419440746307373, -0.3125791847705841, -1.5019066333770752, -2.3202571868896484, -0.26259350776672363, -0.44745951890945435, 1.0767525434494019, -1.4792649745941162, -0.45890623331069946, 0.8164843916893005, 0.8715092539787292, -0.3491637408733368, 1.353830099105835, 1.6559637784957886, 0.48377248644828796, 0.5187642574310303, -1.428952693939209, 1.9076881408691406, -0.26029306650161743, -0.5947129726409912, -0.40699446201324463, 0.5154358148574829, -1.0445916652679443, 1.6313302516937256, 0.6391284465789795, 1.284609317779541, 0.8148739337921143, 1.1230260133743286, 0.5108678340911865, 0.18582546710968018, 0.2438248246908188, 0.6405359506607056, -0.28867465257644653, -0.3061474859714508, -1.3617902994155884, -1.1748666763305664, -0.8619030714035034, -1.3136522769927979, 1.539278507232666, 0.1800532191991806, 0.8068190217018127, 0.07626454532146454, -1.4758598804473877, -0.029539739713072777, -0.6196503639221191, 1.36994469165802, -0.46524670720100403, 0.9849894046783447, 0.36595994234085083, 0.0872693732380867, -2.064451217651367, -1.0267986059188843, 1.5617225170135498, -1.7926826477050781, -0.302585244178772, -0.12731096148490906, -0.1407916396856308, -0.09827019274234772, -0.6644694805145264, -0.7809310555458069, 0.9143264889717102, -2.0779166221618652, 0.20674942433834076, 0.6040306091308594, 0.553851306438446, 0.7695126533508301, 0.151977077126503, -1.480929970741272, -0.16374021768569946, 0.33418944478034973, 0.3419463634490967, 0.36503347754478455, -1.0136932134628296, 1.3824362754821777, 0.4624423384666443, 0.2739865779876709, -1.2411926984786987, -0.5808653235435486, -0.4321931302547455, -0.7971565127372742, 0.20591963827610016, -1.7841960191726685, -0.011060542427003384, -1.0917258262634277, -0.528890073299408, 0.8566365242004395, -2.7939555644989014, 1.5755451917648315, 0.2587366998195648, 0.7946289777755737, -0.5928220152854919, 0.2351415455341339, -2.0259923934936523, -1.238125205039978, -0.20303967595100403, 0.12771734595298767, -0.41485148668289185, -0.8696606159210205, -1.2450624704360962, -0.7609267830848694, 1.879367709159851, 0.9459538459777832, 0.9903651475906372, 0.3565191626548767, 1.774624228477478, -0.4129563868045807, 0.5429031252861023, -0.37060049176216125, 0.10428231954574585, 0.1337275207042694, -0.09972816705703735, 0.8759385347366333, -0.4020863175392151, -0.31988033652305603, 0.0647311732172966, 0.7966729402542114, -0.01237565279006958, -1.1015530824661255, 0.028061052784323692, -1.4469788074493408, -0.7871115207672119, 0.9118386507034302, 0.7438663840293884, -1.0947859287261963, -0.6116711497306824, 0.5714631676673889, 1.8301832675933838, -0.8891066908836365, 0.6163319945335388, -0.014211161993443966, 1.4785288572311401, -0.28186118602752686, -0.34354615211486816, -0.6340920329093933, 0.4560565650463104, 0.9405950307846069, 0.5258566737174988, 0.3065364360809326, -0.13784252107143402, 0.5798235535621643, 1.2346999645233154, -0.06052655726671219, -0.14173340797424316, 0.585269033908844, -0.953416109085083, 0.1486414670944214, -1.5574005842208862, -1.271391749382019, -0.02077186107635498, 0.9883084297180176, -0.9844879508018494, -2.0142722129821777, -0.1910017728805542, 1.0676319599151611, -0.09954705089330673, -2.406527519226074, 0.7204809784889221, 1.2633075714111328, 0.2559964060783386, 0.0453459732234478, -0.964614987373352, -2.162590742111206, 0.1263078898191452, 1.1051751375198364, 0.11817573755979538, -0.9327584505081177, -0.39040395617485046, -0.14924825727939606, 1.0088330507278442, -0.2450791746377945, 0.573164701461792, 0.983991265296936, 0.0942573994398117, -0.42471572756767273, -0.28119611740112305, 0.9134562611579895, -0.5764593482017517, -1.0435487031936646, -0.4007375240325928, -0.09686386585235596, -0.42621737718582153, 1.8989354372024536, 0.7219719886779785, 0.7356212735176086, -1.79615318775177, 0.3188498020172119, -0.22279886901378632, 0.01805846393108368, -0.6192219853401184, 0.7067781090736389, 1.2542847394943237, -0.7715683579444885, -0.2682414650917053, 0.027020202949643135, -0.24982918798923492, -1.0824010372161865, -1.5514507293701172, 0.7092780470848083, -0.33182865381240845, 2.1115562915802, -0.13008257746696472, 0.7556575536727905, -0.4658695459365845, -0.6392899751663208, 0.5397958159446716, 0.5366218090057373, -0.7048261761665344, -1.1625374555587769, 0.2397492676973343, -1.2272180318832397, -0.13925276696681976, -0.21786198019981384, -0.07806847989559174, 1.487159013748169, 1.4015809297561646, -0.8176314830780029, -0.5986416339874268, -0.8220589756965637, 0.6754400730133057, 0.1484738141298294, 0.6432953476905823, -0.6710767149925232, -0.05939449742436409, 0.9298885464668274, 1.4082322120666504, 0.34944048523902893, 0.7576228380203247, 0.32228803634643555, -0.8080165386199951, 2.447624683380127, 1.1184569597244263, -1.558109164237976, 0.24815581738948822, 0.4376574456691742, -0.15212945640087128, -0.11161379516124725, -2.093994140625, 0.537736713886261, 0.3254517614841461, -0.3493684232234955, -1.2350581884384155, 1.9030907154083252, 1.1300674676895142, 1.3062015771865845, 0.45868217945098877, 0.7225444912910461, -0.14554418623447418, 1.4225023984909058, -1.5218620300292969, 0.7082667350769043, -0.9695482850074768, 0.9439992904663086, -1.6872690916061401, 0.15161371231079102, 0.6326330900192261, -0.1543896496295929, -0.2642228603363037, -0.46747249364852905, -0.6084039807319641, 0.5766266584396362, 1.9094674587249756, -0.6121645569801331, 0.5005365610122681, -0.4770340025424957, -1.662720799446106, -1.0151046514511108, 2.2667782306671143, -0.5187223553657532, 1.058132529258728, -0.5120894312858582, 0.10722501575946808, -0.9343715310096741, -0.06740136444568634, 1.5716112852096558, -0.5498818159103394, -1.4198251962661743, -0.06455342471599579, -1.8539637327194214, -0.7438735365867615, -0.1456301063299179, 1.0108661651611328, -0.47127479314804077, 2.1676623821258545, 0.7316749095916748, -1.9487541913986206, 0.048587728291749954, -0.34381386637687683, -0.5565725564956665, -0.5796777009963989, 0.1739831566810608, 1.2713818550109863, 0.13442932069301605, -0.38350751996040344, -1.73312509059906, 1.3954089879989624, 0.7260155081748962, 0.4591936767101288, -0.19015023112297058, 1.532309651374817, 0.38936278223991394, 0.9223721623420715, -1.0005886554718018, 0.9364821314811707, -1.5119894742965698, -1.0338877439498901, -0.32861560583114624, 0.9850348830223083, 1.0898290872573853, -0.6495888829231262, -0.7559413313865662, -0.02915828675031662, 1.410573124885559, 1.5405007600784302, -1.0521488189697266, -0.423687607049942, -0.00235732295550406, -0.08297412842512131, 0.5043689012527466, -0.49297162890434265, 0.6385988593101501, 0.7611619830131531, -0.376832515001297, 0.07230190187692642, -0.46494024991989136, 0.7628727555274963, -0.41218581795692444, 0.9036905169487, -0.3604736328125, 0.7122384905815125, -0.8166203498840332, -1.5717072486877441, -0.7645384073257446, -0.31008198857307434, -0.7717730402946472, -0.27024808526039124, 0.7186815142631531, -1.1456868648529053, -1.896787166595459, -1.4807935953140259, 0.44399064779281616, -0.29951173067092896, -1.40218985080719, -0.7435615658760071, 0.9615186452865601, 0.3572712540626526, -0.1887473613023758, 0.6275472640991211, -0.9617502093315125, -0.9329767227172852, -0.01518390141427517, 0.5022125244140625, 0.7213247418403625, -0.9405955672264099, -0.44540640711784363, -1.8568295240402222, -0.416231632232666, -1.0167222023010254, -0.6796577572822571, 0.43475788831710815, -0.885063648223877, 0.6805223226547241, -0.11332547664642334, 0.34713122248649597, 0.7443063259124756, -1.1713542938232422, -1.8981009721755981, 0.24544933438301086, -0.43326058983802795, -0.9561403393745422, -0.4793378412723541, -1.115693211555481, 0.030359279364347458, 0.10375866293907166, -0.25872161984443665, 0.2952204644680023, -0.522286057472229, -0.20106589794158936, -0.6499320864677429, -0.05605018138885498, 0.1453375369310379, 0.5355693101882935, -0.05506483465433121, 1.551748275756836, -0.5172657370567322, 0.29190438985824585, -0.8729346990585327, -0.0552973672747612, 1.1902217864990234, -1.5058039426803589, 0.03820884972810745, -0.26804253458976746, 0.23564867675304413, 0.04352601617574692, 0.1671280860900879, 1.8542497158050537, 0.42176058888435364, -0.6620436310768127, -0.9539119005203247, -0.9056652784347534, 0.9874266982078552, 0.6410323977470398, -1.2053096294403076, 0.2300613522529602, -0.7859873175621033, -0.6446773409843445, 1.6028172969818115, 0.5301807522773743, -1.1344363689422607, -0.6227747201919556, 1.024959921836853, -0.751469612121582, -1.160583257675171, 0.4323314130306244, -1.3218564987182617, 1.5159530639648438, 0.5392642021179199, 0.8948900103569031, -1.902996301651001, -0.7156568169593811, -0.8831515908241272, 0.22292302548885345, -0.38085636496543884, 0.7513655424118042, -0.129411980509758, 0.30971279740333557, -2.7742748260498047, 1.5111621618270874, 0.0866374522447586, -0.6436939835548401, 0.8608400821685791, -0.04272399842739105, 0.19374534487724304, -1.8119109869003296, -1.6282163858413696, 1.9830604791641235, -0.9864974021911621, -1.1602939367294312], [-0.8143755793571472, 0.681881844997406, 0.88435959815979, 1.1319591999053955, -1.4814265966415405, -0.04934629425406456, 1.380851149559021, 0.6790975332260132, 0.2185497134923935, 1.3768551349639893, -1.6896378993988037, -0.8724837899208069, 0.9093733429908752, 0.4611344635486603, -0.9494712948799133, 0.794425368309021, 1.5997041463851929, 0.9617393612861633, -0.13096189498901367, 0.6199299097061157, 1.243382215499878, -0.39976727962493896, -0.0914679691195488, 1.93752920627594, 0.27659428119659424, 0.4542142450809479, 0.7410681247711182, -0.011249950155615807, 0.09269696474075317, -1.1375654935836792, 0.20065928995609283, 1.5309715270996094, 1.1297005414962769, -0.8427932858467102, -2.521080732345581, -0.02448875829577446, 1.1702051162719727, 1.8502622842788696, -1.8095899820327759, 0.7433031797409058, 1.9510760307312012, 1.08762788772583, 0.38500601053237915, 2.083007335662842, 0.9872720837593079, 1.1001296043395996, 0.24908731877803802, -1.3452527523040771, 2.362558364868164, 1.501082420349121, -0.2206493467092514, -0.1990755945444107, 0.7423604726791382, 0.4898504316806793, -1.3219451904296875, 1.4531903266906738, -0.3526363670825958, -0.1278887689113617, 2.5484180450439453, -0.9507811069488525, -0.33838558197021484, 0.4449855089187622, 0.893528938293457, 0.12092424184083939, -1.3199198246002197, 0.20119038224220276, -2.0877325534820557, 0.40090465545654297, 2.0055220127105713, -0.5257781744003296, -0.30762091279029846, 0.5230399966239929, -0.01889338344335556, 0.012779752723872662, 0.7292526364326477, -0.7237062454223633, 1.0290930271148682, -0.07305925339460373, -0.5062246918678284, 1.8999024629592896, -0.6011294722557068, 0.5058545470237732, 1.9750350713729858, 1.0085954666137695, 1.7223516702651978, -1.6727858781814575, -0.5530850291252136, 0.1769241988658905, 0.24944742023944855, -0.4111536145210266, -0.07948530465364456, 1.8615291118621826, 1.5397217273712158, 0.13612326979637146, 0.43938061594963074, 0.08015306293964386, 0.0677858293056488, -1.6342674493789673, 0.42646923661231995, -0.4471709132194519, 0.6644740104675293, -0.401572048664093, 0.1380023956298828, -0.06388657540082932, -0.8851866722106934, -1.2279781103134155, -0.3618345260620117, 0.0005337651818990707, 1.1258373260498047, -0.5118343830108643, -0.33693498373031616, -1.4231977462768555, 2.110536813735962, -0.5413004159927368, -0.3031720519065857, 1.2086209058761597, 0.8704031705856323, -0.19753339886665344, -0.21697750687599182, 1.4684134721755981, -0.3965202569961548, -0.6575629115104675, 1.2707966566085815, -0.10923163592815399, 1.5451427698135376, 0.19709032773971558, -1.290105938911438, -1.4037275314331055, 0.3386811912059784, 1.035216212272644, 0.265804648399353, 0.032042428851127625, -1.150095820426941, -2.114914655685425, -0.2584969997406006, -0.47583311796188354, 1.3698205947875977, -0.44214311242103577, 0.7383025884628296, 0.18663042783737183, -0.8412326574325562, 1.2295633554458618, 0.03919236361980438, -0.1147816926240921, 0.12511925399303436, -0.5375672578811646, 0.8672374486923218, 0.03349946066737175, -0.13431522250175476, 1.5644499063491821, 0.8539178967475891, -0.05744047462940216, 0.8771956562995911, 1.2076853513717651, 0.048657387495040894, 1.1529381275177002, -2.008330821990967, -0.4631175398826599, 0.32131338119506836, 0.2996990382671356, -0.00647474080324173, -0.5439406037330627, 1.6420377492904663, -2.9451491832733154, -0.6624435782432556, -2.089780330657959, -0.9057666063308716, 0.8715736865997314, -0.2888452708721161, 0.13743600249290466, 1.3009042739868164, 0.49859610199928284, 0.2103012502193451, -0.9836475849151611, 0.5009394884109497, -0.9834783673286438, 0.21370218694210052, 1.1260511875152588, 1.068572998046875, -0.8303946852684021, -1.2849103212356567, 0.12996381521224976, -0.3495476543903351, -0.1936231553554535, -0.25276029109954834, -0.09185057878494263, -2.6654446125030518, 2.801500082015991, 1.017005443572998, -0.4986073672771454, -0.30686232447624207, 1.248085618019104, -0.7432876825332642, 0.13647906482219696, 1.365252137184143, -1.0991607904434204, -1.6913681030273438, 0.5499045252799988, -0.7570586204528809, -0.3918408751487732, 1.341357946395874, -1.2670235633850098, -0.8917989134788513, -0.9719123244285583, 0.0804104134440422, 0.3092384934425354, 1.2676950693130493, -0.23845753073692322, 0.4788922965526581, -0.0802193358540535, 0.9934358596801758, -1.6087875366210938, -1.2951085567474365, -0.3954889476299286, 0.7258636951446533, -0.7883781790733337, 0.4582151472568512, -0.058158669620752335, 1.6204723119735718, 0.9392884969711304, -0.17069369554519653, -1.1591030359268188, -1.00520920753479, 0.5614264011383057, 1.194171667098999, -2.651559591293335, 0.29947131872177124, -0.18203651905059814, -1.1185377836227417, -0.8681358098983765, 0.050516437739133835, 0.19185781478881836, -1.5889662504196167, 0.5319656133651733, 0.7410073280334473, 0.12406424432992935, -0.17314574122428894, -1.16415536403656, -0.42347148060798645, -0.4097547233104706, -0.10309421271085739, -0.625442624092102, 0.0018844787264242768, 0.911145031452179, -0.04057236388325691, -1.1418908834457397, -1.5276471376419067, -0.3496238589286804, 0.4174869656562805, -2.01703143119812, -0.03796542435884476, -1.605421543121338, -0.5739439129829407, 0.7657636404037476, 2.737004280090332, 0.6728823781013489, 1.8140854835510254, -1.4397932291030884, -0.6558303236961365, -1.4846590757369995, 1.158947467803955, -0.5905646085739136, 0.10446291416883469, 1.7390835285186768, 1.7878681421279907, 1.4047355651855469, -0.11762888729572296, -1.4693585634231567, 0.8498713374137878, -0.21931369602680206, -0.16084422171115875, -0.8200646638870239, 1.4226707220077515, -1.2386091947555542, 0.8940398097038269, -0.6569622159004211, -1.1600040197372437, -0.20041249692440033, -0.5071792006492615, -0.721161961555481, 1.9009565114974976, -1.1492209434509277, 0.19601640105247498, -0.09163248538970947, 0.4665970206260681, 0.26735150814056396, 0.9698694944381714, 0.06387512385845184, 0.9380848407745361, 0.5195697546005249, -0.274759441614151, 0.11175953596830368, 0.31922072172164917, -0.5410856604576111, 0.9009804129600525, 0.3531842827796936, 0.23584428429603577, 0.7457137703895569, -0.7129490375518799, -0.5611892342567444, 1.6358906030654907, -0.5416197776794434, -1.981855034828186, -1.726724624633789, -0.12020283937454224, 0.5722977519035339, -0.6049305200576782, -0.629815936088562, -1.3131284713745117, -1.4749245643615723, -1.1265305280685425, 1.2311104536056519, -0.7308583855628967, -1.0983836650848389, -0.603732168674469, 1.776370882987976, 1.3032487630844116, -0.6422135829925537, 0.5344390869140625, 1.000120759010315, 0.8354041576385498, -0.38445764780044556, 1.1059445142745972, 0.5626992583274841, -1.3330062627792358, 0.6940453052520752, 1.039839267730713, -1.0092965364456177, -0.20030947029590607, -3.093431234359741, -0.554135799407959, -1.1557011604309082, -1.226232886314392, -0.8619574904441833, 0.03319466486573219, 0.8688642978668213, 1.0223814249038696, -1.5970768928527832, 0.5758376717567444, -1.3034120798110962, -1.3033154010772705, 0.2025432586669922, -0.09633186459541321, 0.3948932886123657, -0.2022637575864792, -1.3832716941833496, -0.25811946392059326, 0.7297875285148621, 0.4731333255767822, -0.18182358145713806, 0.30889952182769775, 0.9885696172714233, -0.3196644186973572, -1.7259377241134644, 0.9733080863952637, -1.320269227027893, 0.6447176337242126, 0.4027276635169983, 0.23566384613513947, -0.1541595458984375, 0.44191187620162964, 0.26051807403564453, -3.0671679973602295, 0.47538840770721436, -1.2992404699325562, 0.33848097920417786, 1.3245540857315063, 1.016200065612793, 0.49619609117507935, 0.43539515137672424, 0.765015184879303, 1.2398265600204468, 1.109830379486084, 0.7323554754257202, -3.1285109519958496, -0.7229563593864441, -0.6482630968093872, 0.20534762740135193, 1.3811570405960083, -1.3746583461761475, 0.20413629710674286, 1.133430004119873, 1.137791395187378, -0.23023614287376404, -2.2845070362091064, -0.8160833120346069, -0.5858253836631775, -1.2872681617736816, 1.1898303031921387, -1.3169177770614624, 1.0162639617919922, -1.1709651947021484, -1.527213215827942, 0.3053452670574188, 0.16945545375347137, -0.011596150696277618, 1.3219236135482788, 0.8499467968940735, -1.2800966501235962, 0.16208676993846893, 0.6519496440887451, -0.17502851784229279, 0.9996395111083984, 0.39210617542266846, -0.016903119161725044, 1.8749498128890991, 0.2041332721710205, -2.747927188873291, 0.3207739293575287, -0.27832064032554626, 0.9465172290802002, -0.8742418885231018, 0.6984511017799377, -0.04217994585633278, -0.26853498816490173, 0.10014062374830246, 1.9758597612380981, -1.4365177154541016, -0.3957779109477997, 0.26244181394577026, -0.8988999128341675, -0.42196929454803467, -0.8627487421035767, 1.4553827047348022, 0.2786644697189331, -0.05503951385617256, -0.8228535056114197, 0.20319251716136932, 0.26675382256507874, 0.28012892603874207, 1.571134328842163, -0.6673263907432556, -0.21971452236175537, -0.7599362134933472, 1.178764820098877, 1.7244113683700562, 0.48532795906066895, -1.3784117698669434, 1.739538550376892, 0.24162107706069946, -0.4957400858402252, 0.04972830414772034, -1.1908578872680664, -0.6556863188743591, 0.24666297435760498, -0.24993368983268738, -0.13219018280506134, -1.4630268812179565, 1.1933094263076782, 0.7041803002357483, -1.3180991411209106, -0.4986875653266907, -0.3365252912044525, 1.0505484342575073, 0.4783538281917572, -0.36621999740600586, -0.974122166633606, 0.7273425459861755, 0.9726311564445496, -1.1357089281082153, -0.024138983339071274, -0.14158128201961517, -0.07023666054010391, 0.5597514510154724, -0.7977574467658997, -0.7700787782669067, 0.6076752543449402, -0.0894840657711029, -3.034799098968506, 0.27554598450660706, -0.053119827061891556, -1.5845359563827515, -1.588374137878418, 1.496625542640686, 0.7892539501190186, 0.6096513867378235, 0.4221634566783905, 0.7987726926803589, 0.8492143154144287, 0.254652738571167, -1.683253526687622, -1.3730428218841553, 0.638447105884552, -0.42032158374786377, 0.13238103687763214, 1.5325403213500977, 0.18574783205986023, -0.8348361849784851, -1.3591101169586182, 1.155011773109436, 0.6685769557952881, 1.196917176246643, 1.372438669204712, 1.001810908317566, -0.22001540660858154, 0.46303871273994446, -0.37754422426223755, 0.4300982356071472, 0.376168429851532, 0.5706287026405334, 0.21029947698116302, 0.4644855856895447, 0.1606011986732483, 1.4239314794540405, -0.08334443718194962, 1.2755248546600342, 0.8343921899795532, -0.4427930414676666, 0.9303443431854248, 1.2403823137283325, 0.2799570560455322, 1.1969611644744873, 0.04833551496267319, 0.006682927720248699, -0.13399925827980042, -1.5627573728561401, -0.5250425338745117, 1.6599774360656738, -0.925872802734375, 1.564687728881836, 0.7449579238891602, 1.9519392251968384, 0.35974422097206116, 0.9574494361877441, 1.1515042781829834, 0.7654011845588684, -0.8213219046592712, 0.9221518039703369, 2.6037192344665527, 1.2627530097961426, 2.2096362113952637, 1.0826408863067627, 0.2004612684249878, 0.8951515555381775, 0.824381411075592, 0.5923381447792053, -0.050632771104574203, -0.5645684003829956, 0.02932353876531124, -0.8350988030433655, 1.5323536396026611, 0.8391202688217163, 0.4831535518169403, -1.641698956489563, -0.31401342153549194, -0.9919745922088623, 0.2968093156814575, 2.0775012969970703, -1.863966703414917, -1.2211610078811646, -1.0208004713058472, -1.6965885162353516, 0.5148923397064209, -1.3938060998916626, -0.05894048511981964, -0.6578965187072754, 0.41663625836372375, -0.2661316692829132, -1.1135505437850952, 2.0357096195220947, -0.15111970901489258, -0.1580977886915207, 0.6095893979072571, -0.21444270014762878, -1.0788829326629639, 1.147879958152771, -0.42230069637298584, 2.3666412830352783, -0.006663084030151367, 0.3674013912677765, -0.09370674937963486, 0.18433088064193726, 0.1357278823852539, -0.8252766132354736, -0.044972002506256104, 1.1947766542434692, -0.4597582221031189, 0.429226279258728, -1.0276939868927002, 0.2108769416809082, 0.4349019527435303, 0.2562311291694641, -1.1168642044067383, -2.5590462684631348, 2.182436466217041, 0.0629805326461792, 0.343267023563385, -0.20799899101257324, -2.3445534706115723, 1.122869849205017, -1.820945143699646, 0.06769263744354248, 0.46277672052383423, -0.6188117861747742, 0.5213288068771362, -1.0607922077178955, -0.06783971190452576, -1.0858219861984253, 0.7014743685722351, -0.6582653522491455, -1.3079203367233276, 0.7275970578193665, 0.45461514592170715, 0.7344009280204773, 0.7731800079345703, 0.6079628467559814, -0.026458993554115295, -0.9625642895698547, 1.3191142082214355, 0.917962908744812, -1.3686046600341797, 0.135965496301651, 0.5270538926124573, -1.085415005683899, -0.6917827129364014, -0.2133147120475769, 0.5991673469543457, -1.0501693487167358, -1.3613049983978271, 0.1669340878725052, -0.4688371419906616, -0.06891005486249924, -1.1403037309646606, 0.21210531890392303, -0.3358195424079895, -1.1804423332214355, 0.09980203956365585, 1.0448004007339478, -2.2614223957061768, 0.5631746053695679, -1.8688119649887085, 2.3311216831207275, -2.2900516986846924, 1.8734225034713745, -1.125985860824585, -0.46685200929641724, 0.5464616417884827, 0.03731818124651909, -0.30126821994781494, -1.5470004081726074, -0.9555558562278748, -0.027604790404438972, -1.5101139545440674, -0.20104630291461945, 1.0572925806045532, -1.0535788536071777, 0.8412607908248901, -0.7323864102363586, -1.7361347675323486, -0.8612648248672485, 1.5714725255966187, -0.2411079853773117, -0.3597400486469269, -1.6644573211669922, -0.37645766139030457, -0.7107064723968506, 1.3052217960357666, -1.8747897148132324, -0.758594274520874, -0.5348671078681946, 0.3187013268470764, 1.2445811033248901, 0.37308356165885925, 1.6819684505462646, 0.16847583651542664, -0.8570663928985596, 1.3610540628433228, 0.10418332368135452, 0.27256494760513306, 0.8630334138870239, 0.5532026886940002, 0.912770688533783, -0.004570932127535343, 1.4497953653335571, 0.7186100482940674, -1.2004203796386719, -1.5799388885498047, -0.0764300748705864, 0.12721100449562073, -0.15284214913845062, -1.2066792249679565, -0.9005618691444397, 0.2481016367673874, 0.31819459795951843, -0.19048382341861725, 1.85174560546875, 0.9096870422363281, -0.6238811016082764, -0.9777383208274841, -0.5316480994224548, 0.08410239964723587, -2.0651323795318604, -0.3015817701816559, -0.2200217992067337, 1.6728955507278442, 1.9309844970703125, -0.4173394739627838, -1.0433170795440674, -0.3191652297973633, 0.7129977345466614, 0.1369699388742447, -0.010252216830849648, -1.1547656059265137, -0.15363749861717224, 0.05925925821065903, -0.8686122298240662, 0.35863134264945984, -0.15101107954978943, -0.41035914421081543, -1.289283037185669, -0.12605972588062286, -0.5246723294258118, -0.47583314776420593, -1.6327707767486572, 0.03933201730251312, 0.45883095264434814, 0.47686266899108887, -0.5178138017654419, 0.7096964716911316, 2.824425220489502, -0.8029135465621948, 0.5906827449798584, -0.9433099031448364, -1.8217262029647827, 0.7751068472862244, 0.6942545175552368, -0.18841400742530823, -0.5658663511276245, -0.5884548425674438, 1.0815058946609497, 0.9693467617034912, -1.6740754842758179, -1.299445629119873, 0.6694519519805908, 0.02196419984102249, 0.1002042219042778, -0.13931618630886078, -0.6019924283027649, -1.4464092254638672, 0.42226338386535645, 0.47202977538108826, 0.006655240431427956, 0.0013491237768903375, -0.62935870885849, 1.056522011756897, -1.470270037651062, -0.0022898782044649124, 0.11853311955928802, -0.14578209817409515, 2.016587734222412, -1.098343849182129, -1.3876346349716187, -1.04558265209198, 0.9750356674194336, 0.5387052297592163, 0.10288799554109573, 0.13053783774375916, 0.6251681447029114, -1.1087960004806519, 0.02925105392932892, -0.6186914443969727, -2.7185492515563965, -0.543821394443512, -1.1387782096862793, -0.192258819937706, -0.6428573727607727, 0.4021458923816681, 0.09077835083007812, -0.3179154694080353, 1.8870936632156372, 0.8249338865280151, 1.8585447072982788, -0.31324872374534607, 0.5944847464561462, 1.6384207010269165, -0.6190572381019592, 1.2641675472259521, -0.08496176451444626, -1.6051911115646362, 1.1342456340789795, 0.6787887811660767, 1.0904414653778076, 0.07162124663591385, -1.364359736442566, 0.4621843695640564, -0.1600414663553238, 1.330581784248352, 0.2935945391654968, -0.6068655252456665, 1.4333264827728271, 0.1854693740606308, 0.7564775347709656, 0.15496478974819183, 1.7805079221725464, 0.5440145134925842, -0.1261749416589737, -0.6004641056060791, -0.581875205039978, 1.881507396697998, -1.7111215591430664, -0.0032022343948483467, -1.1375263929367065, 0.8050582408905029, 0.4300043284893036, -1.28058922290802, -1.680679202079773, -0.7447537779808044, 0.5242084264755249, -1.580850601196289, 1.0834603309631348, 0.22148627042770386, 1.1610773801803589, -0.6103326678276062, 2.0266497135162354, -1.0827528238296509, 1.3546594381332397, -0.5340133309364319, -0.9258042573928833, -1.380623459815979, 0.5598395466804504, 0.4539172351360321, -0.22680653631687164, 0.6452082395553589, -0.12892885506153107, 1.0377895832061768, -0.591408371925354, 0.670347273349762, -0.2964872121810913, 0.9417630434036255, 1.0695263147354126, 0.3356105387210846, 0.9447460770606995, -0.012545588426291943, 0.7750573754310608, 0.4754472076892853, 1.0521899461746216, -1.8246492147445679, -0.6079873442649841, -1.2569464445114136, 0.1482788771390915, -1.7650604248046875, 0.10804199427366257, -0.9020339846611023, 0.09069346636533737, -0.46450138092041016, 0.523965060710907, 0.7521305680274963, -1.300189733505249, 1.5943132638931274, -0.2921505868434906, -1.1295802593231201, 0.6254820823669434, 0.2022647261619568, -0.7606511116027832, 0.7071275115013123, 0.47314414381980896, 0.6805444359779358, 0.08869460970163345, 0.2594350278377533, -1.3278782367706299, 0.27228182554244995, 1.7769999504089355, -0.5361809134483337, -0.058898963034152985, 0.03325296938419342, -1.1502779722213745, 0.004827512428164482, 1.8621680736541748, 3.13118314743042, -1.3634271621704102, -0.4806935489177704, 0.7362327575683594, -1.0935404300689697, 0.11524023860692978, -0.951825737953186, 2.139273166656494, 2.385141134262085, 0.5881618857383728, 0.3825617730617523, -1.739452838897705, -0.31210508942604065, 0.18267689645290375, -0.8793002963066101, 0.8914963603019714, -0.7946637868881226, -1.5172070264816284, 0.6600608825683594, 0.18228141963481903, -0.7943369746208191, 2.2439093589782715, 1.7561811208724976, 0.6950271129608154, 1.3306453227996826, -1.5846729278564453, -1.0945000648498535, 1.363228440284729, -0.0723850354552269, -2.3964428901672363, -1.153639554977417, 0.9416922926902771, 1.6158936023712158, -2.312950849533081, -0.7779694199562073, 0.7753639817237854, -0.2630506157875061, -1.6750770807266235, -0.11506941169500351, 1.0679093599319458, 0.3044299781322479, 1.117113709449768, 0.09683406352996826, 0.06004989519715309, 0.1568393111228943, -1.0502201318740845, 0.8004140853881836, 0.7993717789649963, 1.3931941986083984, 0.33681580424308777, 0.41234347224235535, 2.570862054824829, 0.8520729541778564, -1.1121915578842163, 1.040823221206665, -1.8052619695663452, -0.2580385208129883, 1.362754464149475, 1.2821348905563354, -0.4636710286140442, -0.9105017185211182, 1.2498118877410889, 0.10298649221658707, -0.8521776795387268, -0.15407812595367432, -2.1750123500823975, 1.2282769680023193, 0.7928797602653503, -0.17569231986999512, 1.806911587715149, -0.5330907702445984, -0.6969753503799438, 0.604789674282074, 0.16550365090370178, -1.6949256658554077, -0.5309020280838013, 1.615999460220337, 0.568932831287384, 1.306356430053711, -0.948558509349823, 0.5018495321273804, 1.7280611991882324, 0.33140459656715393, 0.9400224089622498, -0.36383265256881714, 0.9246183633804321, 0.3132106363773346, -1.8627221584320068, -1.8793866634368896, 0.6939871907234192, 0.3067381680011749, 0.3658803105354309, -0.8125704526901245, -0.4066770076751709, 1.835995078086853, 1.1458525657653809, -1.1043622493743896, -0.3925865888595581, -2.3246960639953613, 0.7906687259674072, 1.0879796743392944, 0.3388221263885498, -0.32979893684387207, -1.0517892837524414, 1.0176640748977661, 0.9752495884895325, 1.2218108177185059, 1.0063238143920898, -0.320203959941864, 0.7100281119346619, 0.8191415667533875, 0.16015373170375824, -0.08846855908632278, -0.5932835340499878, -2.11856746673584, -0.04797732084989548, -1.6988219022750854, -0.325344979763031, -0.4992867112159729, -0.07174793630838394, 0.7492364048957825, 0.6359747052192688, -0.7844161987304688, -0.7585789561271667, -0.9000462293624878, -0.3623754680156708, 0.41902652382850647, 1.125044584274292, -1.3048726320266724, 0.57709801197052, -0.8558406829833984, 0.07556255161762238, 0.77773517370224, 0.23380902409553528, -0.2850479185581207, 0.9561635255813599, -1.9303890466690063, -1.171821117401123, 0.46258097887039185, 0.6527542471885681, 0.12924186885356903, 1.0257997512817383, -0.16974405944347382, -0.2620009183883667, -0.6392453908920288, 2.346111536026001, 0.12544764578342438, -0.6136424541473389, 0.6726194024085999, 1.5569511651992798, -0.011405800469219685, 0.04038963466882706, -1.756209373474121, 1.5174745321273804, -1.126842737197876, -0.6495961546897888], [0.4374934732913971, -0.09569088369607925, 0.7907333374023438, -0.6310284733772278, -1.9253511428833008, -0.14406727254390717, -0.20622850954532623, -1.3981359004974365, -1.4292255640029907, 0.2277134656906128, 0.21036356687545776, 0.329865962266922, 0.354465126991272, 0.07124732434749603, -0.08804947137832642, 0.1664975881576538, -0.7959498763084412, -0.6280783414840698, 1.6531774997711182, 1.948534607887268, -0.6281370520591736, -0.5990746021270752, 0.14851650595664978, 0.8946552872657776, -1.7932724952697754, -0.09543517976999283, 1.1885217428207397, 0.9863790273666382, -0.32417765259742737, 1.94901704788208, -0.17289747297763824, -0.4342319965362549, 1.4115251302719116, 0.021817490458488464, 0.6883257031440735, -2.0605921745300293, 0.07878948748111725, -0.9045880436897278, 1.254773736000061, 2.899235725402832, -1.2731842994689941, -0.33242663741111755, 0.66481614112854, -0.7805838584899902, 0.7892374396324158, -0.0378744900226593, -1.060289978981018, -0.9801429510116577, -0.44031965732574463, -0.9781686067581177, -0.24922141432762146, 0.5548737645149231, -2.9652962684631348, 0.8221094608306885, 0.25983965396881104, 1.470483422279358, 1.4055966138839722, -1.8302984237670898, -1.337799310684204, 0.24623161554336548, 0.5008140206336975, 0.40134987235069275, -0.691677987575531, 0.22831471264362335, 1.1363762617111206, 1.264843463897705, 0.6467422246932983, 1.1263465881347656, 1.5783113241195679, 0.09120479971170425, -0.3726734519004822, -2.184309482574463, 0.13837642967700958, -1.4590997695922852, -0.7956482768058777, 1.7364903688430786, -0.4978075623512268, 0.3217615485191345, 1.5379972457885742, -0.8480362892150879, -0.2596057057380676, -0.453011691570282, 0.9584972262382507, -0.83365398645401, 0.7565652132034302, 0.5579208731651306, -1.3496063947677612, -0.10926695168018341, -0.5405471324920654, 1.0517473220825195, 1.5975176095962524, 1.1053348779678345, 0.9146873950958252, -0.9028478264808655, 1.7866063117980957, 0.19105055928230286, 1.1394665241241455, 0.8909573554992676, 0.6400251984596252, -0.32946285605430603, -1.1013875007629395, -0.6462373733520508, 0.7019347548484802, -1.6007553339004517, 1.0660595893859863, 0.297454833984375, -0.13011674582958221, 1.3673416376113892, -1.2632538080215454, 0.3788289427757263, 0.009138322435319424, 0.9806829690933228, 1.859133243560791, -0.9297231435775757, 0.055847618728876114, 0.5646381378173828, -0.6447117328643799, -0.7395411133766174, 0.13848601281642914, 1.1542974710464478, -1.2230751514434814, -0.27624791860580444, -0.03980773314833641, -0.13911829888820648, -0.3451754152774811, -0.9192596673965454, -0.8338922262191772, 1.1267985105514526, -0.7383742332458496, -0.24097464978694916, -0.7634084224700928, -0.4016699194908142, 0.8112028241157532, 1.8850473165512085, 0.5033653974533081, 1.0872999429702759, -0.8758959770202637, 1.8857539892196655, 0.9844157099723816, 0.6333237886428833, -1.9314770698547363, -0.504563570022583, -0.40748676657676697, -0.16429874300956726, 0.5049101710319519, 1.5722129344940186, 0.6324036717414856, -1.688054084777832, -0.779938280582428, -0.6754341125488281, 0.5807509422302246, 1.633371353149414, 1.9242355823516846, 1.315101146697998, 3.4404513835906982, 0.3930811882019043, -0.09434637427330017, 0.3033818006515503, -0.986018180847168, -0.8226481080055237, 1.9475648403167725, 0.7487301826477051, 0.34714606404304504, 1.4683080911636353, 0.9343497157096863, 1.3504984378814697, 0.7845490574836731, -0.7695661783218384, -0.015631666406989098, -0.12425628304481506, 1.8022706508636475, 0.5269467234611511, -1.107426643371582, -0.11049813777208328, -0.37535133957862854, -0.07376933097839355, 0.819252610206604, -0.46981725096702576, 0.08623617142438889, 1.4401031732559204, 0.7415251731872559, -0.21349328756332397, 0.5205954313278198, -2.596987009048462, -0.47552087903022766, 0.7574250102043152, 0.5505858063697815, -0.2597483992576599, 0.22497902810573578, -0.014960643835365772, 0.10823702812194824, -0.6821550130844116, 0.4492950737476349, -0.9130127429962158, -0.5637816190719604, -0.32544776797294617, -0.07260206341743469, -0.14985817670822144, 0.5651799440383911, 0.23078913986682892, 0.35406455397605896, 0.7673775553703308, 1.0300971269607544, -0.3142206072807312, 1.5991026163101196, 0.766386866569519, 0.48343539237976074, 0.8971865773200989, 1.00038480758667, -0.6163033843040466, 1.0136677026748657, -0.7360069751739502, -0.5068200826644897, 1.1959308385849, 0.5656522512435913, 1.1295756101608276, -0.43004176020622253, -0.9168336987495422, -1.6620396375656128, 1.5175652503967285, -0.05375588685274124, -0.4630475342273712, 1.462119221687317, 0.7548718452453613, -0.07221134752035141, 1.2269761562347412, 0.30184975266456604, -1.358027696609497, 1.7243669033050537, -1.6038738489151, 0.5613465309143066, -0.8816102147102356, 0.3445530831813812, 1.8412108421325684, -1.1954007148742676, -0.33758318424224854, -0.5096145272254944, -0.7754770517349243, 0.4264998137950897, -1.6210006475448608, 0.9948107004165649, 2.3388545513153076, 0.09149599075317383, -1.255110263824463, 0.3778208792209625, 0.7297771573066711, 1.571866512298584, -0.14938990771770477, 1.0490055084228516, 0.6885180473327637, -0.7756885886192322, -1.3464163541793823, 0.06903677433729172, -0.38320493698120117, 0.7435013651847839, 0.18403714895248413, -0.8237156271934509, -0.03352859988808632, -0.2451065480709076, -1.5043607950210571, 1.6577736139297485, -1.9733048677444458, -0.4349777102470398, 1.0782841444015503, -2.004676580429077, 0.8483686447143555, 1.8467963933944702, -0.18577854335308075, -0.19620132446289062, -0.6279873251914978, -0.11790314316749573, -0.42562490701675415, 1.6780288219451904, -0.05252038314938545, 0.867051362991333, 0.10288625210523605, 1.0578358173370361, 0.9726438522338867, -0.011006557382643223, -1.255062460899353, -0.09772992879152298, -2.3253557682037354, -1.519166350364685, -0.522716224193573, -0.18688926100730896, -0.6159685254096985, -0.5169997215270996, -1.8532335758209229, -1.546416163444519, 2.0145785808563232, -0.48487263917922974, 0.04497000575065613, -0.6442496180534363, 0.6253550052642822, 2.603553056716919, 0.38626474142074585, -1.935431718826294, -0.7670383453369141, -0.025138497352600098, 1.2564326524734497, -0.013492261990904808, -0.3464825451374054, 0.3350337743759155, 1.6129800081253052, 0.33046114444732666, 0.7778584957122803, 0.4436309039592743, 0.6522736549377441, 1.3242427110671997, -1.616780400276184, -1.497024416923523, 0.3702009618282318, 0.9029189348220825, -0.28132864832878113, -1.1049280166625977, -0.0013736773980781436, 0.9252226948738098, 0.2844710946083069, 1.1648205518722534, -0.5681654214859009, -0.4350075125694275, -0.9792115688323975, 1.2369872331619263, 1.0462818145751953, -0.32013338804244995, 0.3006955087184906, -0.09134922921657562, -1.5212594270706177, -1.363533616065979, 0.11847580969333649, 0.12622670829296112, -0.05338480696082115, -1.3210995197296143, -1.321210265159607, 1.3591994047164917, -0.15833429992198944, 1.4517353773117065, 0.3673534095287323, -0.5477306246757507, 0.6213123202323914, 1.5564168691635132, 0.6933465003967285, 0.20389191806316376, 0.18094152212142944, -0.10198410600423813, -0.6800947785377502, 1.0527921915054321, 0.3371385633945465, -0.6130343079566956, -0.2890808880329132, -0.40738797187805176, -1.068021535873413, 2.0325124263763428, -1.123028039932251, -0.5737484693527222, 1.1282398700714111, -0.09156738221645355, 0.7076107859611511, 0.7616273164749146, 1.4583629369735718, -0.5000991821289062, -1.00710928440094, 0.8419986963272095, 0.565019965171814, 0.920487642288208, -1.6721293926239014, 0.7744142413139343, -0.5672658681869507, -0.14467142522335052, -0.3897291421890259, 0.027353543788194656, -0.18787015974521637, 2.9159326553344727, 2.7342305183410645, -0.593777060508728, 0.6721430420875549, -0.14608363807201385, 0.17792245745658875, 0.19582712650299072, -1.3783986568450928, 1.153396725654602, 0.6818005442619324, 0.9155415296554565, -0.2520945966243744, 1.1378045082092285, -0.09100396931171417, -0.5418518781661987, -1.2088778018951416, 2.2061290740966797, -0.7911040782928467, 0.9306023716926575, -1.3533049821853638, -2.172606945037842, -1.119762659072876, -1.4028328657150269, 0.8396366834640503, 1.8641144037246704, 0.29851841926574707, -0.9202535152435303, 0.8945584893226624, -1.210870385169983, -1.0732378959655762, -0.7172150611877441, 1.456154704093933, -0.8129915595054626, 0.2769440710544586, 1.0688714981079102, -1.801797866821289, -1.2324010133743286, 1.6619398593902588, 0.3780573904514313, -0.3239041268825531, -1.2927569150924683, 0.380371630191803, 0.358328253030777, 0.26518478989601135, -0.48199462890625, 0.9069671630859375, 0.5695364475250244, 0.31925326585769653, -0.44307324290275574, 0.07948736101388931, 0.06640323251485825, -0.8967079520225525, 0.7734606266021729, -1.5069937705993652, -0.19600845873355865, -0.47411665320396423, 0.36468175053596497, 0.09659954905509949, 0.7721226811408997, -1.41172194480896, -0.3867279589176178, -1.1730722188949585, -0.33073776960372925, 0.6709651350975037, -0.26466840505599976, -0.8388661742210388, 0.3161409795284271, -1.019189715385437, 1.0358532667160034, 0.14409154653549194, -1.1141706705093384, 0.8134384751319885, 0.45847928524017334, 0.17233982682228088, 1.1609957218170166, -0.9617339372634888, 0.47375890612602234, 0.3769886791706085, 0.3066493272781372, -0.20189879834651947, 1.2941380739212036, 0.8450265526771545, 1.3614007234573364, -0.42209702730178833, 0.23945152759552002, 0.3018145263195038, 1.6797133684158325, 0.11460579186677933, -0.7173717021942139, 0.21652452647686005, 1.2782796621322632, 1.000792384147644, 0.012608742341399193, 1.824554204940796, 0.8271306157112122, -0.40243852138519287, -0.6779995560646057, 1.2012174129486084, -0.29147395491600037, 0.03360116109251976, 1.7778161764144897, -3.005432605743408, 2.064440965652466, 1.385618805885315, 0.7227650284767151, 0.36571836471557617, -0.8438869118690491, 2.625626802444458, 0.9737195372581482, 0.8442246317863464, -0.5896188616752625, 0.2962917685508728, 0.7486794590950012, 0.3078550696372986, 0.13233034312725067, -1.6418445110321045, -1.2473901510238647, 1.967469334602356, -0.5318835973739624, -0.008805298246443272, -0.709048330783844, -0.23657728731632233, 0.960162341594696, 1.4525377750396729, 0.2656633257865906, 0.2160642296075821, -0.11449547857046127, -0.2541704773902893, -0.9903501272201538, 0.2439737468957901, -1.2824993133544922, 0.7884657979011536, -0.5897817015647888, -0.8582732081413269, 0.23093976080417633, -0.8559662103652954, -1.103705883026123, 0.09833712130784988, -1.7066395282745361, 0.12041538953781128, -0.7528945803642273, 1.1880638599395752, -0.3056783974170685, -0.6508070230484009, -0.13263268768787384, -0.9134694933891296, -2.3743884563446045, -1.744336485862732, 0.4234169125556946, -0.8359156250953674, -0.08424888551235199, 0.12546725571155548, -0.13498437404632568, -0.19300682842731476, -0.121436208486557, -1.8376144170761108, -0.11075595766305923, 0.5571475028991699, -0.831632137298584, -2.2063004970550537, 1.131108045578003, 1.7251312732696533, 1.0631396770477295, 1.0333411693572998, -0.5151260495185852, 0.7115596532821655, -0.09894739091396332, 0.2928789258003235, -1.3455256223678589, -2.274588108062744, 1.3308889865875244, 1.4997973442077637, -0.3077191114425659, -0.4135722219944, 0.7983713746070862, -0.1873784363269806, 0.1675156056880951, -0.3725281357765198, -1.2487542629241943, -0.17018091678619385, -0.2735479176044464, -0.04726439341902733, 0.06956584751605988, -0.2981525659561157, 1.2702239751815796, 0.06627286970615387, 0.2623933255672455, 1.399017095565796, 0.12252756953239441, -0.07979851961135864, 0.5504234433174133, -1.8088454008102417, -0.4913407862186432, 0.7611532211303711, 1.9028372764587402, -0.36449122428894043, -0.6282443404197693, -2.5552895069122314, 1.0386508703231812, 0.0515320710837841, -0.5795181393623352, -0.35256481170654297, -0.24140988290309906, 0.3177397549152374, 0.5590565800666809, -0.04131539538502693, -0.6767327785491943, -0.19945767521858215, 1.1309772729873657, -0.01650618016719818, -0.34138211607933044, 0.14886431396007538, -0.1340678632259369, -0.2586853504180908, 1.1287957429885864, 0.7316291332244873, 1.5216180086135864, 1.7700684070587158, 0.3702934682369232, 1.0026298761367798, 0.054568905383348465, 1.0908139944076538, -0.9519541263580322, -0.4289919137954712, -0.7200297713279724, 2.119971513748169, 0.5230354070663452, -1.9314002990722656, -0.354280948638916, 0.6407523155212402, 0.48865652084350586, 1.421006441116333, 0.34881871938705444, -2.3769371509552, 0.3366776704788208, 1.6602405309677124, 0.5281840562820435, 0.6797791123390198, 0.06715378910303116, -0.06295215338468552, 0.3717442750930786, -0.9828539490699768, 1.428853154182434, 0.43603092432022095, 1.1339735984802246, -1.8200452327728271, -1.6511081457138062, 0.8452547192573547, -1.1925727128982544, 0.0825682207942009, 0.36044400930404663, -0.07282159477472305, -0.4416155219078064, 1.4372217655181885, -0.3400264382362366, -0.4426664113998413, 0.4023241698741913, 0.7239421010017395, -0.3034757971763611, 0.34767287969589233, 0.9504650235176086, -0.4833151698112488, -1.9122817516326904, -1.8103901147842407, -0.4611199200153351, 0.4468494951725006, 1.2985987663269043, -1.0829976797103882, 0.32550978660583496, 0.49993547797203064, -0.6698981523513794, 0.454894095659256, 0.6039835214614868, 0.36413562297821045, -0.011808968149125576, -0.37891682982444763, -1.7056783437728882, 0.10386725515127182, 1.063330054283142, -0.6415091753005981, -1.930861473083496, 0.1785939633846283, -1.8372172117233276, -0.9782339930534363, 0.5065554976463318, -0.5480679869651794, -1.294037103652954, 0.13059617578983307, -1.3749061822891235, 0.2503237724304199, 0.7031722068786621, -0.8121578097343445, -1.0779415369033813, -0.39956134557724, -0.8309398889541626, 0.46452397108078003, 0.22760607302188873, 1.353262186050415, 0.6571908593177795, -1.3450528383255005, -0.9198130369186401, 0.7528645992279053, -0.07595493644475937, -0.9748485088348389, -0.7447828650474548, -1.882665753364563, 0.16823168098926544, -0.8865350484848022, -0.8858389258384705, -1.5562721490859985, -0.9134923815727234, -1.4396100044250488, -1.1693894863128662, -0.2613816559314728, -1.6711925268173218, 2.132596731185913, -0.5844863653182983, -1.5819429159164429, -0.6827138662338257, 0.4354274570941925, -0.7380977272987366, -0.6171382665634155, 0.7101125121116638, -0.39997929334640503, 1.261448860168457, -0.366949200630188, -0.2321678102016449, -0.16427546739578247, -0.5558733344078064, -1.1849502325057983, -0.5026876926422119, -2.32332444190979, 2.2446868419647217, -2.679914951324463, -1.1080081462860107, -1.2299671173095703, 1.7105586528778076, 0.638824999332428, -0.7707484364509583, -0.8331683278083801, 1.3922162055969238, -0.3103232979774475, 1.9385253190994263, -0.7971700429916382, -0.5478565096855164, -2.0536794662475586, 1.9774179458618164, -0.17053957283496857, -0.3073720633983612, 0.037114158272743225, 0.4672220051288605, -1.048766016960144, 0.15264128148555756, -0.5146101713180542, 1.4130064249038696, 1.2610892057418823, -0.7318116426467896, 0.00748476991429925, -0.6854366660118103, 0.9846622943878174, 0.6061621308326721, 0.6251521110534668, -1.1003779172897339, -0.2969755530357361, 0.3065691590309143, -0.2256758213043213, 0.9224284887313843, -0.26105865836143494, -0.5363944172859192, 1.4751126766204834, 0.7728775143623352, 0.2792046368122101, 1.3131994009017944, 0.30781468749046326, -0.0018318850779905915, 1.5994068384170532, 0.29569199681282043, -0.8702877759933472, -0.8344243168830872, 0.9969556331634521, 0.5179439187049866, -0.6968740820884705, -0.9243540167808533, 1.0025724172592163, -1.2638376951217651, 1.8805830478668213, -1.4500590562820435, 1.6002390384674072, 0.9241529703140259, -1.143497347831726, -0.7568977475166321, -0.008923253044486046, 1.9700359106063843, -0.39815792441368103, -0.08160759508609772, -1.0749194622039795, -0.252320259809494, -0.37469223141670227, 0.51134192943573, 0.8830678462982178, -0.07254426181316376, 0.21127671003341675, -1.894127368927002, 0.08734800666570663, 0.47043710947036743, 0.32612818479537964, -1.0163018703460693, -0.41996997594833374, 0.3368479907512665, -0.9254418611526489, 0.5869895815849304, 0.7249060869216919, -1.4911506175994873, 1.3796104192733765, -1.3740371465682983, 2.3030292987823486, -0.8006223440170288, -1.05973482131958, 0.11252864450216293, -0.6421942114830017, -1.3312610387802124, 0.2891004681587219, 0.3302205801010132, -1.8915164470672607, 0.30738383531570435, 0.14358334243297577, 0.26455891132354736, -1.368411898612976, 1.2224220037460327, 1.0554124116897583, -1.056885838508606, 0.40533989667892456, 0.8135444521903992, -0.6755975484848022, 0.9129335880279541, -0.08833061158657074, -1.030777096748352, -0.10753156989812851, 0.32353055477142334, 0.2590172290802002, 0.9312727451324463, 2.208704710006714, 1.7037209272384644, 0.7764509320259094, -0.259117990732193, 1.32961106300354, -0.14962178468704224, -0.2106836438179016, -0.08727268874645233, 0.4013197124004364, -0.7877183556556702, -0.8341273069381714, -0.09638573974370956, -1.0833518505096436, 0.5048800110816956, -0.38867515325546265, -0.5801094770431519, -0.3135435879230499, 0.1045895516872406, 0.6113186478614807, 1.3948335647583008, -0.5651757121086121, 1.0391432046890259, 1.1977300643920898, 0.04761210456490517, 2.17620587348938, -0.10428261756896973, -0.5120798349380493, -1.7280197143554688, -0.5897645354270935, 0.8636636734008789, -1.3168978691101074, -1.5818901062011719, -1.9020276069641113, -0.2644922137260437, 0.3666357696056366, 0.0025694367941468954, -0.7568899989128113, 0.44472429156303406, -1.8552662134170532, 0.1920251101255417, 0.013572532683610916, -0.8244239091873169, -0.24368621408939362, 0.7600398063659668, -0.2722959518432617, -0.10030908137559891, -0.9070992469787598, -2.1822803020477295, -0.2401113659143448, 0.13837945461273193, -0.0712493360042572, 0.5830287337303162, -2.4243555068969727, -2.0843260288238525, 0.8527197241783142, 0.07834544777870178, 0.7482491731643677, -0.9751514196395874, 0.8607673048973083, 0.41322606801986694, -0.7941681146621704, 1.656151533126831, -0.6704179048538208, -0.9051283597946167, -2.1474132537841797, -0.10144259035587311, 0.5102685689926147, 0.7245785593986511, -1.1553198099136353, 1.4129170179367065, 0.3836895525455475, -0.6813391447067261, 1.0264760255813599, 0.5040162205696106, -0.14876990020275116, -0.3768240809440613, 1.5384446382522583, -0.8332551121711731, 0.1257726550102234, -0.6641517281532288, 1.4568402767181396, 2.556628465652466, 0.1095159500837326, -1.3474456071853638, 0.9007743000984192, -0.5315260887145996, -1.4151540994644165, 0.03423859924077988, -1.3857842683792114, 0.12486481666564941, -0.30515021085739136, 0.5044118762016296, -0.9238699674606323, 1.4569921493530273, 1.4337594509124756, 0.3010958731174469, -0.17058052122592926, 2.5342071056365967, -0.5325301885604858, 0.14681664109230042, 0.9937211871147156, 0.10654769837856293, 1.5120296478271484, 0.48554420471191406, 1.7506132125854492, -0.35793742537498474, -0.7762974500656128, 0.6390287280082703, 0.3369672894477844, 1.7516069412231445, 0.9205509424209595, -1.4743093252182007, 1.3919751644134521, 0.47851303219795227, 0.16315706074237823, 0.5506287813186646, 1.4953739643096924, 0.6272462606430054, -0.3334354758262634, 1.0735726356506348, 1.0000537633895874, -0.5159321427345276, -0.07774567604064941, 1.2204811573028564, 0.7363072633743286, -0.7153931260108948, 0.4108365774154663, 0.8447697758674622, 0.625155508518219, -1.289494514465332, 0.31014180183410645, -0.3994004726409912, -2.18648362159729, 0.3113384246826172, 0.3821103870868683, 1.4675331115722656, 0.8732088208198547, -1.113611102104187, 0.17202582955360413, -0.792140543460846, 0.21473008394241333, -0.28207480907440186, -0.26381441950798035, 1.4954330921173096, 0.47795426845550537, -0.9035136103630066, -0.04279600456357002, -0.4319942593574524, -0.10828286409378052, 0.9142168760299683, -0.122264064848423, -0.13763673603534698, 0.14320075511932373, -0.7597882151603699, -0.5711533427238464, -1.8064757585525513, -1.6530535221099854, 0.6044220924377441, -0.26673394441604614, 1.2507208585739136, 0.17723582684993744, -0.029656853526830673, 0.041737403720617294, 0.08308107405900955, -0.5144941210746765, 0.2743605375289917, -0.18425773084163666, -1.5452802181243896, 0.12895193696022034, -1.4717910289764404, 0.5714345574378967, 0.023321926593780518, -1.9217101335525513, 0.6958379745483398, 0.2363121658563614, -1.1237125396728516, 0.2403666377067566, -0.2833021581172943, 1.4400995969772339, -0.007708228658884764, 0.9406691789627075, 1.644120216369629, -0.03607816994190216, 0.4353753626346588, 2.396667242050171, -0.45709460973739624, -1.7296311855316162, -1.1668579578399658, -0.35296377539634705, 0.21523690223693848, 0.18410056829452515, -0.2642085552215576, 0.6671368479728699, 0.1423579454421997, -0.5290817022323608, -0.3758845925331116, -1.1143876314163208, -0.37862786650657654, -0.22870978713035583, -0.9940792918205261, -0.0026787484530359507, -0.9390192627906799, -0.9489558339118958, -0.5531978011131287, -0.40402865409851074, 0.9241912364959717, -0.6225735545158386, 1.104333758354187, 1.6610229015350342, 0.5194686651229858, -0.05157944932579994], [-0.14850948750972748, -0.8073511123657227, 1.8675355911254883, -0.39653342962265015, -0.8625344038009644, 0.9570360779762268, -1.2245147228240967, 0.4936788082122803, -0.35925042629241943, 0.24064922332763672, 0.7112777829170227, -0.03525238856673241, 0.15908464789390564, -1.4706286191940308, -1.1942843198776245, 0.5565218925476074, 0.34601402282714844, 0.9555454850196838, 0.3077029287815094, -0.26651644706726074, -0.9488400816917419, 1.0306538343429565, 1.9736384153366089, -0.92153000831604, -0.31537681818008423, -0.4684585630893707, 2.0913619995117188, 0.24168209731578827, 0.8653374314308167, -0.008596053346991539, 0.33205339312553406, -1.2033488750457764, -0.2090953290462494, 1.1608933210372925, -0.5806170701980591, -1.2215372323989868, 1.5278514623641968, 0.8286339640617371, -1.3221945762634277, 0.206751748919487, 1.041447401046753, 1.4367399215698242, 1.43684720993042, -1.5050618648529053, 0.10466396808624268, -0.73138827085495, -0.324257493019104, -0.3235233724117279, -0.8292037844657898, -0.6541376709938049, -1.5333622694015503, -1.2517610788345337, -1.0503379106521606, -2.7023863792419434, -0.5004606246948242, -0.9330387115478516, -0.6405378580093384, -0.2561292350292206, 1.3678721189498901, -0.12634924054145813, 0.9805413484573364, -0.11849766969680786, -2.224248170852661, -0.4060864746570587, -1.3804218769073486, -0.7638779282569885, 0.2918933928012848, -1.0501127243041992, 0.7746652364730835, 2.154794692993164, -0.14183224737644196, -0.8697183132171631, -0.7650442719459534, -0.07527877390384674, -1.0339778661727905, -1.0356149673461914, 0.4332970380783081, 0.8534023761749268, -0.3161660134792328, 1.0727424621582031, -0.09508457779884338, 0.7994440197944641, -2.3460285663604736, -0.6968429088592529, 1.1267621517181396, -2.686037302017212, 0.3538667559623718, -0.15031667053699493, 0.22042107582092285, -0.5969029068946838, 0.20101435482501984, -0.9214511513710022, -1.3654541969299316, -0.24241763353347778, -0.2978793978691101, 0.13512463867664337, 1.1893762350082397, -1.3084919452667236, 0.9449492692947388, -0.5952723622322083, 0.8782503008842468, 0.8892267942428589, 0.5683103203773499, 0.8130886554718018, 1.5786693096160889, -0.920741617679596, -0.029036995023489, 1.1496407985687256, -0.400680273771286, 0.3689887225627899, -0.7721800804138184, -0.5051583051681519, 0.8305065631866455, 0.26474741101264954, 0.28569626808166504, 0.8218433856964111, -0.05002741888165474, -0.6105971336364746, 0.3248853087425232, 0.3700549602508545, 0.23885053396224976, -0.8847925662994385, 0.9260595440864563, 0.2660679817199707, -0.6853131651878357, -0.08014856278896332, -1.4813979864120483, -0.14965908229351044, 0.09148473292589188, -0.8435084819793701, -0.3057250380516052, 0.05474596843123436, -0.2802620232105255, -0.08796019107103348, -0.46126076579093933, 0.8027500510215759, -0.12317538261413574, 0.7456167340278625, -1.0782023668289185, 0.32664716243743896, 0.06993379443883896, 0.44325417280197144, 0.16672445833683014, 1.5287929773330688, 0.1155484989285469, -1.2309006452560425, 0.6272141337394714, 1.4670101404190063, -1.0944126844406128, -0.03979881480336189, -0.022557541728019714, 1.06890869140625, -0.47136443853378296, 1.6795496940612793, -0.30437928438186646, -1.3356468677520752, 0.47298458218574524, 0.15007361769676208, 1.1662211418151855, 1.6946685314178467, -0.34595227241516113, -0.05681474879384041, -0.4538286030292511, -0.9914254546165466, 0.3823750615119934, 0.8592272400856018, -0.5263078212738037, -0.7621877193450928, 1.0126525163650513, 1.1949199438095093, 0.03503705561161041, -0.11511930078268051, 1.049557089805603, 1.053444266319275, -1.2119776010513306, -0.7183695435523987, 0.6912145018577576, -0.2814692258834839, -0.6128793358802795, -1.7764689922332764, 1.7138793468475342, -0.03337555006146431, -0.8777902722358704, -0.884635329246521, -0.5623975396156311, 0.8737933039665222, 0.06875026226043701, 0.20476175844669342, 0.7368107438087463, -1.9593275785446167, 1.1735974550247192, 1.8220463991165161, -0.4055284261703491, -0.866576075553894, 1.3233892917633057, 1.7370470762252808, 0.5523896813392639, -2.8839752674102783, 0.15097227692604065, -1.1423670053482056, 0.05266983062028885, 1.239549160003662, -1.2978169918060303, 0.2576616108417511, -1.0259898900985718, 0.33325234055519104, 2.438518762588501, 0.5869553089141846, -0.3790861964225769, -1.8918830156326294, 0.14822572469711304, 1.1590101718902588, -1.4704536199569702, -0.24750956892967224, -0.919167697429657, -0.5894086360931396, -2.032632827758789, -0.8442070484161377, -0.27885591983795166, 1.225733757019043, 0.6777889132499695, 0.48676127195358276, 0.5732821822166443, -0.7355314493179321, 1.9182392358779907, 1.6488182544708252, 0.09626661241054535, -0.5697621703147888, -0.4933971166610718, 0.39760565757751465, 0.910080075263977, 1.6477038860321045, 0.6713823080062866, -1.637886881828308, -0.19282428920269012, 0.43431615829467773, 0.22361522912979126, 1.8651599884033203, -1.0674364566802979, 0.9880761504173279, 0.8304593563079834, -0.6864991188049316, -0.8806696534156799, 0.09499139338731766, -0.07711539417505264, 0.8848361372947693, -1.0357708930969238, -0.34785181283950806, 0.3705061376094818, -0.7241942286491394, -1.1063196659088135, -1.1247941255569458, -0.40878674387931824, 0.29584723711013794, 0.6950165033340454, 0.6692942976951599, 1.1425737142562866, -1.1988681554794312, 1.882896900177002, 0.6090176105499268, -0.7505092024803162, -1.1357018947601318, -0.39979851245880127, -0.30944526195526123, 2.932096242904663, 0.4607393145561218, 0.286410927772522, 2.372445583343506, -0.8934086561203003, 0.017894607037305832, 0.6138692498207092, -1.4646507501602173, 0.8744430541992188, 2.031275987625122, 0.5084641575813293, -0.3727794885635376, -1.1045961380004883, 0.8502973914146423, -0.3512427806854248, 1.1930896043777466, -1.0450631380081177, 0.7312927842140198, 0.800054669380188, -1.6373027563095093, -1.118270993232727, 1.138728380203247, 0.30937686562538147, 0.3748302459716797, -0.8072614669799805, 0.948871910572052, -0.03815840929746628, -0.47986674308776855, 0.4474228620529175, 0.03989563137292862, -1.1295664310455322, 1.125108003616333, -0.7949294447898865, -1.232893466949463, 0.00629310542717576, -1.65211021900177, -0.02246733009815216, 0.9033916592597961, 0.04132848605513573, -0.36160925030708313, -0.8410055041313171, -0.1925722360610962, -1.9445041418075562, -0.04997697100043297, -0.810126543045044, -0.8573777675628662, 2.8898746967315674, -1.0352479219436646, 0.09436299651861191, 0.8888439536094666, -0.728884220123291, -1.5685418844223022, -0.8517197370529175, 0.9338459372520447, 2.1354012489318848, 0.1709977239370346, 0.3101647198200226, 0.5059902667999268, 0.5416291952133179, -0.1289946436882019, 1.294680118560791, 0.09302474558353424, -1.1169599294662476, 1.3673349618911743, 0.7821710109710693, 0.39391234517097473, -0.3100292980670929, -0.39377129077911377, 2.55452299118042, -1.1019434928894043, 0.3887203633785248, -0.26614275574684143, 1.876466155052185, -1.1891852617263794, 0.3301863670349121, -0.21741189062595367, -0.9817013144493103, -0.8190390467643738, 0.5760591626167297, -1.1963975429534912, 0.9521108865737915, -0.7168731689453125, 0.9756544828414917, 0.19050836563110352, 0.42022812366485596, 0.9602559208869934, 0.36315447092056274, 0.6887656450271606, -0.3373781740665436, -0.7450228333473206, -1.1979707479476929, -0.4372901916503906, -0.6074026823043823, -0.7384094595909119, 1.7285759449005127, 0.30189090967178345, -0.5369932651519775, 0.23423144221305847, -0.8435084223747253, -0.3038889467716217, -0.7309000492095947, 0.46543142199516296, 1.2235581874847412, 0.7618051171302795, -2.355257511138916, 0.42589014768600464, -1.2020701169967651, 0.13746361434459686, 0.10910660773515701, 0.24515241384506226, 2.1389782428741455, 1.078985333442688, -0.9174981713294983, -1.280124545097351, 0.3549140989780426, 0.7065275311470032, -0.4738186299800873, -1.1948210000991821, -1.0214394330978394, -0.887803316116333, 1.36773681640625, 0.9611454010009766, -0.7945346832275391, -0.15099190175533295, -0.32278281450271606, 1.4703296422958374, -1.3338638544082642, 1.481791377067566, 0.4076853096485138, 0.5410927534103394, -1.012000322341919, 0.21972310543060303, -0.27979010343551636, 0.33512595295906067, 1.0572354793548584, 0.9078058004379272, 1.1119680404663086, 0.48431938886642456, -0.14193129539489746, 0.7992047667503357, 0.3806115388870239, -1.5092122554779053, 1.9059714078903198, 0.6020331382751465, 0.3269994258880615, 1.0298352241516113, 0.2544499635696411, -0.7239640355110168, -0.12165713310241699, -0.11597654968500137, -0.35577473044395447, 2.9685845375061035, -1.0611815452575684, 0.26138854026794434, 0.5871623754501343, -0.8288419246673584, 0.505672812461853, -1.686366081237793, -0.2389623075723648, -0.7241962552070618, -0.46976134181022644, 1.746555209159851, -0.8053879141807556, 1.4566199779510498, -0.8354263305664062, 0.3086813688278198, 0.2607036828994751, 0.3531409204006195, -0.1481781303882599, -0.13076245784759521, 0.2609202265739441, -1.2980966567993164, 0.04846545681357384, -1.6353269815444946, -0.5937599539756775, 0.7848235368728638, -0.07449730485677719, 1.5661702156066895, -0.20205646753311157, 0.5614557862281799, -0.869480550289154, -1.1223772764205933, -0.15328861773014069, 1.4386110305786133, 1.1727995872497559, -0.9324966073036194, 0.5079265236854553, -0.19453468918800354, -0.4259847402572632, 1.010744571685791, -0.1481655389070511, 0.9171462655067444, -0.9944487810134888, 0.765450119972229, 0.0006223366362974048, -1.359142541885376, 0.3302055597305298, 0.5193842053413391, -0.005546625703573227, -1.0485305786132812, -0.6365185976028442, 0.827982485294342, -0.43740367889404297, -0.1769014149904251, -0.8531846404075623, 0.2423296719789505, -0.09225596487522125, -0.7191945314407349, -0.15498167276382446, 3.123478651046753, 2.4166104793548584, 0.31932175159454346, -1.8962435722351074, 0.2448713481426239, 0.15919789671897888, -0.18147160112857819, -0.7375622391700745, -0.7729101181030273, -1.11521315574646, -0.5657560229301453, 2.636983871459961, 0.05873798206448555, 0.9217408895492554, 0.015745878219604492, -0.910845935344696, -2.0843665599823, 0.9493854641914368, 2.0851221084594727, 0.16777333617210388, 0.0872218906879425, -1.1914432048797607, 1.0214428901672363, 0.010784255340695381, -0.38393521308898926, -0.806463897228241, -1.0882267951965332, -0.23115427792072296, 0.895533561706543, 1.5151580572128296, 1.5745298862457275, -0.26208651065826416, 0.10599453747272491, -1.085286021232605, 0.10867857933044434, -0.2884833812713623, 0.6958742141723633, 0.1800498366355896, -0.18455000221729279, 1.0954880714416504, 0.11333290487527847, 1.555020809173584, 0.1408776342868805, 0.6390909552574158, 0.8444659113883972, 0.18856847286224365, 0.4753503203392029, 0.028345448896288872, 0.5269430875778198, 0.46955302357673645, 0.7772611975669861, -0.40998104214668274, 1.2726693153381348, 0.2376694232225418, 1.4698542356491089, -0.3463336229324341, -0.9889886975288391, -1.6782869100570679, -0.13402412831783295, 1.6218092441558838, -2.462279796600342, -0.9042288661003113, 0.5801020860671997, 2.0819711685180664, -0.8461283445358276, -0.4338378608226776, -0.39263060688972473, -0.2991175651550293, 0.7298232913017273, -0.33779099583625793, 0.37509289383888245, -0.7414674758911133, 0.9136472344398499, 1.0976669788360596, -0.3123475909233093, -3.0846571922302246, -0.1108785942196846, 0.7850034832954407, -1.9348628520965576, 0.5994818210601807, 0.20453235507011414, -0.7127124071121216, 0.9683703184127808, -0.4720275402069092, 0.5959057211875916, 1.642595887184143, -0.7338812947273254, -0.37607479095458984, -0.04608900845050812, -0.8632596135139465, 0.29905396699905396, 0.50177001953125, -0.5352568626403809, 0.45199355483055115, 0.5538984537124634, -0.8086360096931458, -0.08290856331586838, -0.8892362117767334, -0.004850972909480333, -0.4196903705596924, -1.21027672290802, -0.6387041211128235, -2.699601411819458, 1.5433861017227173, 1.4950021505355835, 0.04430077597498894, -0.2858443856239319, -0.7936815619468689, 1.0868991613388062, -0.41025739908218384, 0.12317939102649689, -0.12102815508842468, 0.40561041235923767, -1.4782710075378418, -0.16284917294979095, 1.1059907674789429, -0.97588711977005, -0.19426167011260986, 0.11316567659378052, -1.4968113899230957, 0.266044020652771, 1.4558770656585693, 0.9838196635246277, 1.3724055290222168, -2.3713972568511963, 1.7137000560760498, 0.924477756023407, -0.8436506390571594, -0.07844153791666031, 0.1875455230474472, 0.26782262325286865, -0.08687233179807663, -0.02501576580107212, 0.6021541357040405, -0.6501846313476562, 1.8376883268356323, -0.016049856320023537, 0.6344512104988098, -0.27303189039230347, 0.9820259213447571, -1.5157989263534546, 0.8843144774436951, -0.05338002368807793, -0.584940493106842, -0.30801719427108765, 1.4149196147918701, 0.19750210642814636, 1.9343312978744507, -0.8640417456626892, -1.5247355699539185, -1.530716061592102, 0.4913122355937958, -1.3597829341888428, -0.13645817339420319, 0.17109577357769012, 0.3023069500923157, -0.20208071172237396, 3.0269410610198975, 1.1121267080307007, 1.5695785284042358, 2.0694541931152344, 1.42496919631958, 0.695263147354126, -0.7105264663696289, -0.7811209559440613, 1.2257601022720337, -1.50922691822052, -0.46931055188179016, -1.2196871042251587, 0.997579038143158, 2.7858669757843018, 0.491949200630188, -0.7063854932785034, 0.9447188973426819, 0.60060715675354, 0.9112768769264221, -0.3936188220977783, -0.08508097380399704, -0.7116021513938904, -0.8587104082107544, -0.9556199908256531, 0.6654657125473022, -1.2109183073043823, 0.8835604786872864, -0.0906621515750885, -0.5541115403175354, 1.2406033277511597, -2.38923978805542, 0.8348948359489441, -0.6331713795661926, -1.4934091567993164, -1.392270565032959, 1.490513563156128, -1.5156519412994385, -1.7405128479003906, -1.5350697040557861, 0.6229361891746521, 1.3331830501556396, -0.8992137908935547, 0.9444695115089417, -0.06591013073921204, 0.9551330804824829, 0.3933669626712799, -1.4752167463302612, -0.15361657738685608, -1.1155799627304077, -0.8820868730545044, -1.4523396492004395, 0.7947828769683838, -2.007443428039551, -1.3459949493408203, 1.0385513305664062, -0.6071094274520874, -1.239970088005066, 0.6581304669380188, -0.2980223000049591, 0.02385668270289898, 1.3013864755630493, -1.4241887331008911, 2.8805558681488037, -0.11494575440883636, -0.5418017506599426, -0.9942260980606079, 2.1791348457336426, 0.2811802327632904, 3.2624876499176025, 0.3301919102668762, -0.006849306169897318, 0.2222762554883957, -1.6430566310882568, -0.29429417848587036, -0.7790698409080505, -0.8542319536209106, 0.40792152285575867, 1.3740925788879395, 0.33918437361717224, 1.2174806594848633, 0.8469858169555664, 0.8900963068008423, -0.40617966651916504, -0.19176793098449707, 0.19556045532226562, -1.2328077554702759, -1.0431913137435913, 1.1428072452545166, -0.9610608816146851, -1.0388096570968628, 1.2374941110610962, 1.5754873752593994, 0.3011654317378998, 0.22682008147239685, 0.9768095016479492, 1.4088490009307861, 0.8164148926734924, 0.351946622133255, 0.5417797565460205, -0.6659249067306519, 0.4029040038585663, -1.215494155883789, -1.1155753135681152, 1.7909531593322754, -0.8770471811294556, 0.11449510604143143, -0.24220921099185944, -0.7342911958694458, 0.7350226640701294, -0.3855808973312378, -1.6865891218185425, -0.1203766018152237, -0.44924822449684143, 0.03143163397908211, 0.4817171096801758, 1.5587772130966187, -0.3306165039539337, -0.33482038974761963, 2.2760469913482666, -1.5459129810333252, -0.4751102030277252, 0.45428672432899475, -0.4444199502468109, 0.880508303642273, -1.7904131412506104, -1.8801947832107544, -0.5415061116218567, -1.278572678565979, 1.616119384765625, 1.389038324356079, -0.4301687180995941, -0.7288094162940979, 0.6439404487609863, -2.484403610229492, 0.4860578775405884, -0.1256491243839264, 0.7981050610542297, -2.1701903343200684, -0.2682991027832031, 0.4587315618991852, -0.3238452076911926, -0.11209297180175781, -0.9158837795257568, 0.883309543132782, -0.3778989613056183, -1.198452353477478, -0.6963990926742554, -0.7775918841362, 2.1787991523742676, 0.41689690947532654, 0.9777386784553528, 0.1125999391078949, 0.30960002541542053, -0.6444147825241089, 0.7141140103340149, -0.6226153373718262, 0.6269555687904358, -0.45073848962783813, -1.6961952447891235, 0.48877575993537903, -0.6603485941886902, -0.11878588050603867, -0.29104378819465637, -1.3470631837844849, 2.8552498817443848, -0.6564378142356873, -1.8085496425628662, 0.5252054929733276, -0.19150769710540771, 0.5693471431732178, 0.12234210968017578, 0.8183606863021851, 2.4411864280700684, 1.2976961135864258, -0.776185154914856, -0.02930142544209957, -0.912820041179657, -0.5721877217292786, 0.8700476288795471, 0.9553940296173096, 0.20654188096523285, 0.8369244337081909, -0.18674887716770172, -0.9594905376434326, -0.5978749990463257, 0.5635673403739929, 0.3600912094116211, -0.05805150046944618, 1.4429566860198975, 0.7636063098907471, 1.131758213043213, 0.7103747725486755, 0.29042625427246094, 0.06668096780776978, -0.031238745898008347, -1.220581293106079, -0.37215638160705566, 0.9392478466033936, -0.3596108853816986, 0.6388811469078064, -0.45163968205451965, 1.455116629600525, 0.757597029209137, 0.6996631026268005, 0.5864095687866211, -2.5202157497406006, -0.42562201619148254, 0.9690196514129639, -0.657024085521698, -0.697870671749115, -0.5975815653800964, -1.6062297821044922, -0.49392613768577576, 0.5693617463111877, 0.9631990790367126, -3.2611052989959717, 1.0553486347198486, 0.5878499746322632, -0.11517730355262756, -0.6191964149475098, 0.29688602685928345, -0.792570948600769, 0.41733866930007935, 0.9106238484382629, -0.9358819127082825, 0.4246232509613037, -0.7514880299568176, 0.37317177653312683, 1.328061580657959, 0.14795880019664764, -0.1587013602256775, -1.1292222738265991, 0.2575598657131195, -0.35623878240585327, 1.401314377784729, 1.4254748821258545, -0.30454522371292114, 1.4415215253829956, -0.6473220586776733, -0.7029381394386292, 1.425455927848816, -0.26444676518440247, -0.44182077050209045, -0.40030670166015625, -0.796165406703949, -0.07172521948814392, 2.219141721725464, 1.5652267932891846, 0.5039516687393188, -0.11125736683607101, -0.9968328475952148, 0.7195313572883606, -0.3509160876274109, -0.251777708530426, 0.6936500668525696, 0.20914241671562195, -0.4712969958782196, -1.3109787702560425, -0.4328368008136749, 0.5070319175720215, -3.4278554916381836, -2.1130211353302, 1.4721921682357788, -0.7843445539474487, -0.000231856873142533, -2.020317316055298, 0.6005586385726929, -1.0005743503570557, 0.11654296517372131, 2.0045433044433594, -0.5089475512504578, 0.5265366435050964, -0.9306787252426147, -0.5342196822166443, 1.4304044246673584, -0.04560229554772377, -0.831074595451355, 0.48737743496894836, -0.15700674057006836, 0.4658031761646271, -0.6446037292480469, -1.2236682176589966, -0.9819157123565674, 0.17078900337219238, 0.5097525715827942, -0.5145643949508667, 1.6366451978683472, 0.3776918649673462, 0.19865350425243378, -0.7727563977241516, -1.9494091272354126, -0.2816537618637085, 0.22059927880764008, 0.42911508679389954, -1.937438726425171, -0.06482334434986115, -1.352288842201233, -0.7096983194351196, 0.7702702283859253, -0.361436665058136, -0.8197181224822998, -0.45163848996162415, 2.2428359985351562, 0.5286859273910522, -1.3628140687942505, -0.313676118850708, -1.103747844696045, 0.11637746542692184, 1.360306739807129, -1.0484986305236816, -0.5040425062179565, -0.3820495009422302, 1.2930760383605957, -0.3911891579627991, 0.06770411133766174, 0.28343379497528076, 1.0906221866607666, 0.17026180028915405, -0.48567885160446167, 0.8896295428276062, -1.4668707847595215, -2.0000927448272705, -0.4743604362010956, 1.2034022808074951, 1.2790287733078003, -0.4471060335636139, -0.6617304682731628, -0.09837549179792404, -1.7918415069580078, 0.30535250902175903, -0.9031510949134827, -0.32664069533348083, 0.4251786172389984, -0.17174291610717773, -0.17990490794181824, -0.08422296494245529, 1.4220601320266724, -0.36878085136413574, -0.09623196721076965, -0.30040934681892395, 0.05800933763384819, 1.149335503578186, 1.2754713296890259, 1.1407647132873535, -1.0323683023452759, 0.17892159521579742, -0.5818601250648499, -1.3943079710006714, 1.7494168281555176, -0.5331899523735046, 1.2544353008270264, -0.8244367241859436, 0.9817153811454773, -0.6404935121536255, -0.09614297747612, 0.2607286274433136, 2.8453752994537354, -0.4878668785095215, -0.572479248046875, -1.7944204807281494, -0.6568692326545715, 0.2957923114299774, -0.37773919105529785, -1.5280269384384155, -1.3713972568511963, 1.4410831928253174, 0.751925528049469, -0.08335061371326447, 0.9198204874992371, 0.7492693066596985, 1.6543422937393188, -0.5803728699684143, -0.9692130088806152, 0.029795976355671883, -0.4193595051765442, -0.358819842338562, -0.3258725106716156, -0.69197016954422, -0.4530479609966278, -0.9178099036216736, 0.2571576237678528, 0.45573410391807556, -1.0687386989593506, 0.517806887626648, -1.239964485168457, -2.3253166675567627, -1.012718915939331, -1.4531117677688599, 0.7232293486595154, -0.5389147996902466, -0.4458323121070862, -0.09381213039159775, -0.9015012383460999, -0.9259666800498962], [0.7957926988601685, -0.1823667585849762, 0.048662811517715454, 0.3494133949279785, -1.3641579151153564, -1.5244179964065552, -1.8029217720031738, -2.5262792110443115, 0.7912283539772034, 0.5353730320930481, -0.8222762942314148, 1.9717222452163696, 0.29537856578826904, -0.8250129222869873, 1.6929141283035278, -1.0003411769866943, -0.7972338795661926, -0.8233752250671387, 1.4362133741378784, -0.7438907623291016, 0.4742797911167145, 2.432603597640991, 0.5420973896980286, 0.6621020436286926, -1.4604984521865845, 0.6291406750679016, -1.73354172706604, -0.6941033601760864, 0.8792781829833984, 2.284362554550171, -1.431774377822876, -0.4571368396282196, -1.485134482383728, -2.4372940063476562, 1.7065449953079224, -1.739650845527649, 0.681006669998169, -0.9688344597816467, 0.3438601493835449, 0.9748310446739197, 0.25957414507865906, 0.044071946293115616, 0.1783740520477295, -0.4321272671222687, 1.4033939838409424, 1.4591832160949707, 2.341076374053955, -1.554767370223999, -0.2845876216888428, -0.3359729051589966, -0.4906637370586395, -1.4671189785003662, 0.7685709595680237, -0.9446931481361389, -1.077939510345459, -1.635399580001831, -1.4527204036712646, -1.2495957612991333, -0.18601271510124207, 0.7477792501449585, 0.21593472361564636, -0.6782148480415344, 1.1954710483551025, 0.09558843821287155, -0.04743417352437973, -0.3960818946361542, 2.66304087638855, -0.9963564276695251, 0.6015815734863281, -0.9646531939506531, 0.8404421210289001, -0.7063522338867188, -2.9223358631134033, 1.7081023454666138, -0.45549827814102173, 1.6154351234436035, 0.19593925774097443, 2.30688214302063, -1.1511003971099854, -0.9592535495758057, 0.839188814163208, -1.8388190269470215, 1.4295713901519775, -1.253572702407837, -0.32864242792129517, 1.0319945812225342, 1.0509332418441772, -0.566627025604248, 0.6558790802955627, -0.23165728151798248, -1.1505659818649292, -1.2237398624420166, -0.17671170830726624, 0.18386247754096985, 0.7231289744377136, -1.4285635948181152, 0.5731749534606934, -1.009679913520813, 0.22105786204338074, -0.22319462895393372, 0.04198477789759636, 1.030231237411499, -0.30618828535079956, -0.20556771755218506, -0.4923458695411682, 3.06396222114563, -0.7203006148338318, -0.14182668924331665, -0.20173342525959015, -0.9568776488304138, 0.8195925951004028, 0.9845935106277466, -0.6166281700134277, -0.3969959318637848, 0.7312885522842407, -0.18908752501010895, 0.9644474387168884, 0.8917950391769409, 0.46098077297210693, 0.46387791633605957, -0.24240434169769287, -1.104761004447937, 0.4440232217311859, 0.34767335653305054, 0.21326516568660736, 0.5723361372947693, 0.1089152842760086, -0.9083762764930725, -0.3045251667499542, 0.5336057543754578, 0.8635270595550537, 1.8675296306610107, 0.4124828279018402, -0.9264456629753113, -0.6047020554542542, -1.0056179761886597, 0.39879193902015686, 1.2914714813232422, -1.0047943592071533, 0.3196362555027008, 0.10895390063524246, 0.27257785201072693, 0.5069273710250854, 0.9783928990364075, -2.803175926208496, 0.7558196187019348, -0.6877632141113281, -0.7862550020217896, 1.3919962644577026, 0.44804906845092773, -1.1432749032974243, -0.16707159578800201, 0.49618300795555115, -0.19057486951351166, 0.6722444891929626, -0.1363748013973236, -1.3757368326187134, -0.07734811305999756, -0.26206767559051514, -0.3340629041194916, 0.7216768264770508, 0.5869579315185547, -1.1517475843429565, 0.3786737024784088, 0.06754522770643234, -0.41092735528945923, 1.178230881690979, -1.2936350107192993, 0.937725841999054, 0.2621172368526459, -0.04640694707632065, 1.7292060852050781, -1.1690739393234253, 2.1735405921936035, -0.5692934989929199, -0.8861392140388489, 0.7174535989761353, -0.3777357339859009, 0.6158912181854248, 0.5127236843109131, -0.40219151973724365, -0.7811252474784851, -0.648938000202179, 0.8955326080322266, -0.7013857960700989, 0.1660165786743164, 0.9384854435920715, -0.5251452326774597, -0.0019416678696870804, 0.6911640763282776, -0.38881102204322815, 0.8548067212104797, -0.8465814590454102, -0.16967560350894928, 0.9877550005912781, -0.042814865708351135, -0.04207529127597809, 1.047398328781128, -0.12126436084508896, 0.46608439087867737, 1.2565929889678955, 0.21512362360954285, -0.5599012970924377, -0.5228579640388489, 1.0232447385787964, 0.186489537358284, 1.346521019935608, 0.8398762941360474, 0.4817635416984558, 0.9592533707618713, 1.1594507694244385, -1.9913842678070068, 0.8606297969818115, -0.5397520661354065, 0.28921234607696533, 0.20074114203453064, 0.09157229214906693, -0.2608702480792999, -0.2937536835670471, -0.5504235029220581, -1.2147563695907593, -1.3064234256744385, -0.731311559677124, -1.4344582557678223, 0.158927783370018, -1.28795325756073, -0.5739715099334717, -0.8545725345611572, -0.4080800712108612, -0.6652353405952454, -0.34801211953163147, 1.1756887435913086, 1.1224751472473145, 0.7302533984184265, 1.3869843482971191, -0.3518381714820862, -0.9673556685447693, 0.7919504642486572, -0.6491907835006714, -0.7535268068313599, -1.2762356996536255, 0.26746466755867004, 0.2593264877796173, -1.4311621189117432, -1.2405822277069092, 0.4425468146800995, 1.3397669792175293, -0.7453657984733582, -0.00922276359051466, 1.2427161931991577, -0.21529671549797058, 1.3706614971160889, 0.386404812335968, 0.3919045329093933, 0.5988399386405945, -0.9041332006454468, -0.32533884048461914, -1.3584723472595215, 0.5690960884094238, 1.8529012203216553, 0.12603811919689178, -0.07490163296461105, 0.11462206393480301, 0.34252628684043884, -1.7655848264694214, -0.5130409002304077, 0.10859011113643646, -1.9158555269241333, 1.0533391237258911, -1.139312744140625, -1.202874779701233, -0.4096970856189728, 0.05690093711018562, -0.2792123854160309, 1.3121181726455688, 0.6549801230430603, 1.2929422855377197, -0.6870835423469543, 1.7425117492675781, -0.6672388911247253, -0.5305754542350769, 0.3021860420703888, 0.15728411078453064, -0.3675849437713623, 1.1840524673461914, 0.01598644070327282, -0.8286204934120178, -0.654244601726532, -0.44242745637893677, 0.36105379462242126, 0.37280821800231934, 0.20931223034858704, 0.8422664403915405, 0.7708764672279358, -1.8572134971618652, 1.0975064039230347, -0.16640692949295044, -0.3068316876888275, 0.9980791211128235, 0.7496398091316223, 0.3351377844810486, -1.0007988214492798, -0.6283589601516724, 0.8690953254699707, 0.029305122792720795, -0.36016151309013367, 0.22111310064792633, -0.10387396067380905, 1.3797968626022339, -1.3394726514816284, 1.1218016147613525, 1.145538330078125, -0.382728636264801, -1.93397057056427, -0.8491700887680054, 1.898997187614441, -1.6772713661193848, -0.32300981879234314, 0.2947269082069397, 0.40515825152397156, -0.7558879256248474, 0.2325669676065445, 0.19748792052268982, 1.5141290426254272, -1.0103000402450562, 0.2452339082956314, 0.012673940509557724, -1.8848823308944702, -0.9201207756996155, 0.3370032012462616, -0.18525542318820953, -1.3242541551589966, 0.5772107839584351, -1.1464638710021973, -0.5320454835891724, -0.8692618012428284, 0.4355873465538025, -0.3289632499217987, 0.19264917075634003, 0.23626624047756195, -0.7482248544692993, 0.4689483344554901, -1.0487191677093506, -1.3489025831222534, 0.5424846410751343, 1.3246952295303345, -0.9201522469520569, 0.9040839076042175, -1.1007170677185059, 1.3162273168563843, 0.9823554754257202, -0.15126848220825195, -0.08808016031980515, 0.8226485252380371, -0.2784779965877533, -1.0156792402267456, -0.046596672385931015, -0.9810386300086975, 1.1192187070846558, -0.2672012746334076, -0.0541420504450798, 0.9514583945274353, -0.47159186005592346, 0.7070063352584839, -0.06336613744497299, 0.3025491237640381, -0.7221635580062866, -0.2473650872707367, -0.14101175963878632, -2.244389295578003, 0.3857385516166687, 1.2701932191848755, 1.7007101774215698, -1.644979476928711, -0.042040009051561356, -1.0359306335449219, -0.9950867891311646, -0.7238515615463257, 1.3395806550979614, 1.1472643613815308, -1.7599244117736816, 1.0895648002624512, 0.3700869381427765, -1.988067388534546, 0.5863357782363892, 0.5395990014076233, 0.3530518710613251, 0.47566819190979004, -1.1827057600021362, 1.341191291809082, 2.14866304397583, 0.06638671457767487, -0.297748863697052, -0.08382297307252884, 0.8199806213378906, -0.1879092901945114, 2.176327705383301, 0.3872913122177124, -0.3052186071872711, 0.41574472188949585, 1.0176197290420532, 1.2052199840545654, 1.2643675804138184, -0.23757821321487427, 1.004807472229004, -1.5171643495559692, 0.3605218827724457, 0.08109568804502487, -0.7788670659065247, -0.5550475716590881, -0.01045148354023695, 1.1407736539840698, -0.762274980545044, 0.3499509394168854, 0.0476733073592186, -0.4775128662586212, 0.7975876331329346, -1.7560036182403564, 0.6894906759262085, 0.16923648118972778, 1.0232187509536743, -1.2132647037506104, -0.8540223240852356, -0.31089815497398376, -1.385118842124939, -0.5022708177566528, -0.1899467557668686, -0.4141863286495209, -0.8822078108787537, 2.6532833576202393, -0.4600426256656647, 0.14394551515579224, 0.9268568158149719, 0.9240685105323792, -1.2433204650878906, 0.9031702876091003, -0.5465552806854248, -2.1301534175872803, -1.268984317779541, 0.14933791756629944, 0.24774475395679474, -0.25903555750846863, -0.21093542873859406, -1.296274185180664, -2.065505266189575, -1.3851197957992554, 0.7346151471138, -0.2745303809642792, -0.08423013240098953, -1.2157187461853027, -1.2990227937698364, -1.6338003873825073, 1.5203007459640503, -1.508968710899353, 0.561328649520874, 0.608011782169342, 1.1282339096069336, -0.40014952421188354, -1.817417025566101, 1.2258039712905884, -1.455358862876892, -0.7575334310531616, 0.3695089519023895, -0.11997953057289124, 2.110506057739258, -0.18058985471725464, 0.2912891209125519, 0.09454505890607834, -0.2736412286758423, -0.8388195633888245, -0.12277098745107651, 0.21361654996871948, 0.3517446517944336, -0.6130508780479431, -1.1353343725204468, -1.3572441339492798, 0.48460644483566284, -0.0582018606364727, -0.04571583494544029, -0.1851034164428711, -0.3355543315410614, 0.0877724289894104, 0.289827823638916, -0.3272896707057953, 1.3400214910507202, 0.33015507459640503, 1.339339256286621, 0.16633930802345276, 0.1100170686841011, 0.20378170907497406, 0.9780437350273132, 0.3765234649181366, -0.37372398376464844, 0.7529613971710205, 0.21451951563358307, 0.29955482482910156, -1.12409245967865, -1.4347209930419922, -1.419825553894043, 1.927815556526184, -1.4231834411621094, -0.03195175528526306, 0.6395789980888367, -0.6250728368759155, -0.3710483908653259, 0.23771174252033234, 1.0770353078842163, 0.6082918643951416, -0.18180814385414124, -1.5422896146774292, -0.5384266376495361, -0.2539971172809601, -0.7619903683662415, -1.1980974674224854, -0.2841740846633911, -1.4428305625915527, -1.4911227226257324, 0.976475715637207, 0.8400905132293701, -1.5864500999450684, -1.1925594806671143, -0.3543630540370941, 0.3412243723869324, 0.0989576205611229, -0.4289489686489105, 1.173866629600525, 0.5733191967010498, -0.06934680044651031, 0.007815425284206867, -0.6424901485443115, -1.4260677099227905, 1.6596708297729492, 0.44713446497917175, 0.6827647686004639, -1.0324219465255737, -0.2026517242193222, 0.43888574838638306, 1.1726075410842896, -1.417551875114441, -0.4108399748802185, 0.08220718055963516, 1.5036885738372803, 1.4113667011260986, -1.1133294105529785, 1.4541325569152832, -0.06791296601295471, -0.24163207411766052, -1.3430384397506714, -0.4947667717933655, -0.398781955242157, -0.9298636317253113, -0.14601871371269226, 0.4841066300868988, 0.3409120440483093, 0.9735029935836792, -1.1851166486740112, -0.7264593243598938, -0.4292987585067749, -0.2665744423866272, 1.5413230657577515, 0.5446031093597412, -1.0627573728561401, 0.794419527053833, -0.6646258234977722, 0.2866130471229553, -0.5526793599128723, -0.010151958093047142, 1.070311188697815, 0.3093418776988983, 0.06496685743331909, 0.1478043645620346, -1.2254561185836792, 1.4821507930755615, -0.1305450201034546, 0.27629518508911133, 0.6789082884788513, 0.34948602318763733, 0.17058010399341583, 1.0695785284042358, 0.5969704985618591, 0.20135754346847534, -0.23004408180713654, 1.2536327838897705, -1.8552535772323608, -0.03413047268986702, 0.6938924789428711, -1.3880560398101807, 1.4205081462860107, 0.6157605648040771, 0.7536219954490662, -1.4833520650863647, -0.17090271413326263, 0.9975128769874573, -0.42713433504104614, 0.8516333103179932, -0.9888439178466797, -0.641156792640686, -0.7200220823287964, 0.44124335050582886, -0.2831169068813324, 0.05768400430679321, 1.1100152730941772, 1.401539921760559, -2.6799135208129883, -1.3304471969604492, 0.3439590036869049, 0.1491943597793579, -1.5958362817764282, -0.8230751752853394, 0.17307926714420319, -0.29013386368751526, -0.009493042714893818, -7.340281445067376e-05, 0.7674720287322998, -1.1901179552078247, 0.37819787859916687, 0.9073053598403931, -0.9344121813774109, -1.2286335229873657, 2.2031357288360596, -1.8546651601791382, 0.8635445833206177, 0.10930850356817245, 2.7047431468963623, 1.1531858444213867, -0.7620392441749573, 1.6902152299880981, 0.6830026507377625, -1.0223617553710938, 0.3018675446510315, 1.4390721321105957, 0.751642107963562, -0.12561950087547302, 0.8258479237556458, 0.3536135256290436, -0.682633101940155, 0.3069429099559784, 1.0483635663986206, -1.3686386346817017, -0.2439456731081009, -0.7389730215072632, -0.3097640573978424, -0.7463832497596741, 0.6783959865570068, -0.22283083200454712, 0.5413805842399597, 0.7977514863014221, -0.8922857642173767, -0.3289228081703186, 1.4682461023330688, -1.0715067386627197, 0.2274908572435379, -1.0072696208953857, -1.3653610944747925, -1.4590511322021484, 0.5175918936729431, 0.014173822477459908, -0.7458831071853638, -0.7243813872337341, 0.4314124584197998, -0.3952009379863739, -0.8215194940567017, 2.245303153991699, -0.6774466633796692, 1.118888020515442, 0.9637730717658997, -0.14156584441661835, 0.05465605482459068, -1.6251369714736938, 0.4220035672187805, -2.2723536491394043, 0.13854238390922546, 2.045651435852051, 1.7695327997207642, -0.43626290559768677, 1.2002071142196655, 1.3779375553131104, -2.2311830520629883, 0.3150908946990967, 1.1586050987243652, 0.5106822848320007, 0.545382022857666, 1.5622774362564087, 1.6116089820861816, -0.08232179284095764, -0.6498016119003296, 0.3544904887676239, -0.9676218032836914, 0.678180992603302, 0.8629179000854492, -0.24780981242656708, -2.612065076828003, 0.3987918496131897, -1.617458462715149, 0.42984867095947266, 0.6428580284118652, -0.9256151914596558, -0.6967225670814514, -0.1582433581352234, -1.150887370109558, 2.2595674991607666, -0.9199315309524536, -0.6898438334465027, -1.3410606384277344, -0.05006559565663338, 0.5718022584915161, -0.03277760371565819, 2.2187345027923584, -0.07254637032747269, 0.2742985188961029, -0.4238207936286926, -0.7481878995895386, 0.7127708792686462, -1.1771708726882935, -0.16966700553894043, -0.3083849847316742, 1.085982084274292, -1.200167179107666, -0.255525141954422, -0.14406044781208038, -0.6935849785804749, -0.33503323793411255, -1.2115521430969238, -0.14213936030864716, -0.438569039106369, -0.6608191728591919, 0.505082905292511, 0.12214983999729156, 0.22663497924804688, 0.3485793173313141, 0.5058799982070923, -2.110074520111084, -0.9413278102874756, -1.1778761148452759, 0.5528348684310913, 0.3057706356048584, -0.07271692156791687, 0.29915639758110046, 0.10492508113384247, -0.39784273505210876, -0.010290179401636124, -0.4488751292228699, 0.4407314956188202, 0.19067613780498505, -0.21506847441196442, 1.999951958656311, 0.8411925435066223, 0.20139658451080322, -0.7398547530174255, -1.0077768564224243, 0.003318113274872303, 0.37076297402381897, 0.18152843415737152, 0.4464341998100281, 1.3158012628555298, 1.0981789827346802, -0.24448011815547943, -0.13163769245147705, -0.9086564183235168, -0.1406318098306656, -0.26687702536582947, -0.889616072177887, -0.16825294494628906, 0.48486682772636414, 1.8228639364242554, -0.4558757543563843, 1.2574138641357422, -1.1605559587478638, -0.23922835290431976, -0.26194024085998535, -2.994516134262085, 1.2213985919952393, 1.7268633842468262, -1.4792319536209106, 0.7544102668762207, 1.8337408304214478, 0.3946188688278198, 1.3441346883773804, -1.0970971584320068, -1.2880992889404297, -0.15686306357383728, -0.44880402088165283, 0.8222787976264954, -0.15337805449962616, 0.6139947175979614, 0.18783389031887054, -1.4449509382247925, -0.21341772377490997, 0.4893510341644287, 1.4421184062957764, -0.32911989092826843, 0.17102685570716858, 0.8093281388282776, 2.279837131500244, -1.124678373336792, 1.116043210029602, -0.12199479341506958, 0.640350878238678, 0.49098676443099976, -0.8236799836158752, -1.1130777597427368, 0.13976307213306427, -0.6657488346099854, 0.04014264792203903, -0.7553421854972839, -0.6388452649116516, 1.2915382385253906, 1.6394020318984985, 0.7919637560844421, -1.8580925464630127, 1.7075315713882446, 1.1241331100463867, 0.11931343376636505, -0.12190120667219162, -1.5711897611618042, 0.9037263989448547, 0.3104250133037567, 0.4612244963645935, -0.2882528305053711, -0.25527945160865784, -0.9474751353263855, -0.7325493097305298, 0.11403028666973114, -1.268957495689392, -0.15327143669128418, -0.7726030349731445, -0.4701398015022278, -1.5003975629806519, -0.0131308538839221, -1.2200969457626343, -0.7744920253753662, -0.07652271538972855, 0.12690052390098572, -0.1349617838859558, 0.8316750526428223, -0.09187676757574081, -0.5361924171447754, 0.38925793766975403, -0.1854403167963028, 0.0958586037158966, 0.7709525227546692, 1.3188930749893188, -0.4681997001171112, -1.4690499305725098, -0.2959277033805847, 0.45703351497650146, -0.5164799094200134, 0.9788122773170471, 0.17906370759010315, -0.42568057775497437, -2.8823347091674805, 2.7930197715759277, -1.0834519863128662, -0.5698394179344177, 0.2757440209388733, 0.39753666520118713, 0.17478612065315247, -1.6355983018875122, -0.23627321422100067, 0.39259445667266846, 0.224725142121315, -1.911110281944275, 1.1382596492767334, -0.10649387538433075, 1.819586992263794, -1.4058308601379395, 1.148858666419983, -0.07127732783555984, -0.33898043632507324, -0.2317236363887787, 2.4929702281951904, 0.23783482611179352, 1.7206251621246338, 1.4511297941207886, 0.24637538194656372, -0.5847298502922058, -0.24031886458396912, -0.2601562738418579, -0.19315724074840546, 1.3059334754943848, -0.13098859786987305, -0.7083532214164734, 0.3161575496196747, -0.10337653756141663, -0.715198278427124, -0.08795271068811417, 0.38640907406806946, 0.9675642251968384, 0.1628817468881607, 0.23278315365314484, 0.6747367978096008, -0.87514728307724, 0.38706207275390625, 0.15133579075336456, -0.44361868500709534, 0.5041532516479492, 2.049206495285034, -0.37646836042404175, 0.025488294661045074, 0.21597790718078613, -2.076747417449951, -0.0813295766711235, -0.3881415128707886, -0.8243343234062195, 0.6297951936721802, -0.038324322551488876, -1.039208173751831, 0.8970852494239807, -0.754564642906189, 0.5749370455741882, -1.0916097164154053, 0.29403090476989746, 1.8337749242782593, -1.1618210077285767, -1.6790403127670288, -0.6128191351890564, -0.060746531933546066, -1.7288579940795898, -1.244873046875, 0.9983884692192078, -1.2478141784667969, -0.6053195595741272, -0.8949559330940247, 0.5807377099990845, 1.3857890367507935, -0.8184059262275696, 0.4745652973651886, -0.21249856054782867, 1.300095796585083, 1.7810864448547363, -1.3434323072433472, 0.04233183339238167, -1.181713581085205, 1.0222389698028564, 2.191828966140747, -1.2861441373825073, -0.05735126510262489, -0.36716845631599426, 1.4195127487182617, 0.29590123891830444, -0.3799287676811218, 0.8130326867103577, 0.765978217124939, -0.40474963188171387, -0.11105068773031235, -2.1105010509490967, 0.670288622379303, 0.37150824069976807, -0.9596678018569946, -1.8654485940933228, 0.43054768443107605, 0.7925759553909302, -1.0129871368408203, -0.6449211239814758, -2.206338882446289, 0.3893517851829529, 0.7964597940444946, -0.24272048473358154, -1.1912124156951904, -0.31356707215309143, -0.12496597319841385, 1.221612572669983, -0.29864370822906494, 0.8363661170005798, -0.813444197177887, -0.6239609718322754, -0.19340433180332184, -0.7102484703063965, 0.18758003413677216, 2.829836845397949, -1.092644214630127, 1.0746210813522339, 1.2868300676345825, -0.10193168371915817, -0.7249107956886292, 0.5679356455802917, -0.25678497552871704, -0.6712601780891418, -0.6782516837120056, -0.5340920090675354, -1.4489895105361938, 0.21813008189201355, -1.8333820104599, -0.05177321285009384, -0.2859841585159302, 0.5823511481285095, -0.0821615606546402, 0.8674261569976807, 0.2950242757797241, -0.877150297164917, -0.5287166833877563, 0.4432513415813446, -0.4506857097148895, -2.4770915508270264, 0.10962147265672684, 2.64437198638916, -1.0064231157302856, -0.10941093415021896, 1.758979320526123, -0.9034906029701233, -1.9852334260940552, -0.2969716787338257, -0.5154888033866882, -2.7105119228363037, 0.8131008744239807, -0.8529543280601501, -0.5055255889892578, 0.9569346308708191, -0.2925723195075989, 0.6606088280677795, 0.3152453899383545, -0.5846354961395264, 1.1143074035644531, 1.020235538482666, -1.1172256469726562, -0.4366135597229004, -2.1597647666931152, 1.5453989505767822, -0.3488388955593109, 1.596634030342102, -0.2864038646221161, -0.7371116876602173, 0.5678935050964355, 1.612301230430603], [-0.7542756199836731, 2.2060651779174805, 0.8864424824714661, 0.34906691312789917, 0.228319451212883, 0.3228355646133423, 0.440458744764328, 1.1501270532608032, -0.583809494972229, -0.6821200847625732, 0.3038439154624939, -0.7015770077705383, 1.6016813516616821, 1.897629737854004, 0.4348367750644684, -0.5316532254219055, -0.042740777134895325, 0.2565946877002716, 0.42411139607429504, -1.2672998905181885, -0.12844714522361755, -0.14919520914554596, 2.601811408996582, -1.390104055404663, 2.0974483489990234, 0.6885780096054077, 0.12029922753572464, 0.8667095899581909, -1.4849505424499512, 0.5828236937522888, 0.040507491677999496, -0.311679869890213, -1.555657148361206, 0.04196816310286522, 1.049092411994934, -0.7308014631271362, 0.34441956877708435, 0.12099887430667877, -0.5691603422164917, 0.8030405640602112, 0.7517045736312866, 0.45017215609550476, -0.1858627200126648, -0.31412747502326965, 0.15764184296131134, -0.2663205564022064, -0.7054505348205566, -1.9394869804382324, 0.05754326283931732, -0.6893677711486816, 0.9210621118545532, -0.5088388919830322, -1.1156232357025146, 0.8589236736297607, -2.123957633972168, -0.8596816062927246, -1.4132040739059448, -2.3738274574279785, 0.8766198754310608, -1.549065113067627, -1.272776484489441, 1.0272303819656372, 1.0595977306365967, 0.29276928305625916, -0.4112851917743683, -0.5723952054977417, -0.4517911672592163, 0.8008747696876526, -0.031203042715787888, -1.1958526372909546, -0.805718183517456, -0.023929448798298836, -2.640765905380249, 0.5836367607116699, 0.29568925499916077, -0.02921285293996334, 0.6816388964653015, 0.3736632764339447, 0.29090040922164917, 0.3005477488040924, -0.3979635238647461, -1.0450429916381836, -0.45091503858566284, -1.0865813493728638, -2.007714033126831, -0.6112139821052551, -0.8171713352203369, -1.5047329664230347, 1.2926043272018433, -1.4088435173034668, 0.5924558043479919, -0.920201301574707, -1.136806845664978, 0.39765843749046326, -0.0035249993670731783, -0.19019047915935516, 1.2928965091705322, 0.1577211171388626, -0.43927180767059326, -2.101339817047119, 0.6692567467689514, -0.9469209909439087, 1.0233516693115234, -0.16776107251644135, -2.288053512573242, 1.076587200164795, -0.3871156573295593, -0.2191755771636963, -1.0778086185455322, -1.3123270273208618, 0.416814386844635, -0.5324291586875916, 0.14978525042533875, -0.363936185836792, 0.45568764209747314, -0.2462748885154724, -0.6845404505729675, -0.03972882404923439, -0.34430959820747375, -1.2290802001953125, -0.7242959141731262, 0.5979621410369873, -0.04794291406869888, 0.4484034776687622, 0.0018487339839339256, -0.6052657961845398, -1.5341408252716064, -0.5611358880996704, -1.1895126104354858, -1.3227152824401855, 0.5099115371704102, -0.5310412645339966, -0.369327187538147, -0.23783114552497864, 0.03038700856268406, -0.0999751091003418, 0.23665878176689148, 0.8940371870994568, -0.31955885887145996, 1.081524133682251, -1.2202376127243042, -0.1458190679550171, 0.46237924695014954, 0.9298747181892395, 1.3243725299835205, -1.083567500114441, -0.14493006467819214, -0.6414200067520142, -1.839493751525879, 0.3509845733642578, 0.6104652285575867, 1.0195465087890625, 2.17936110496521, -1.7715388536453247, -0.7926455140113831, -0.6052761077880859, -0.5073654651641846, -0.6439148187637329, 0.4587218761444092, -0.038692835718393326, 0.34525978565216064, 0.8537821769714355, -0.4024626612663269, 1.1019078493118286, -0.022322995588183403, 1.6272610425949097, 1.2619225978851318, 0.15566951036453247, 1.0986977815628052, 0.32697591185569763, -0.6995688080787659, -0.08574135601520538, 2.1881375312805176, 0.47496557235717773, -0.3682704567909241, -0.38677841424942017, -0.4627363979816437, 1.2445430755615234, 0.12559804320335388, -1.285726547241211, 0.44962212443351746, -0.8368620276451111, 2.879401445388794, -1.9130446910858154, -0.31432175636291504, 0.6507218480110168, 0.23822224140167236, -0.5091677904129028, -0.8300635814666748, 0.5350475907325745, -0.6222352385520935, 0.5820376873016357, 0.3353103995323181, -1.8919323682785034, -0.7245907187461853, 0.6177079081535339, 0.14753088355064392, -0.4348941743373871, -0.665037214756012, 1.7513536214828491, 1.2961030006408691, 0.828779935836792, -0.44255387783050537, 0.2648201882839203, -1.3797571659088135, -0.46325257420539856, 0.23212465643882751, 0.8380489349365234, -1.174646019935608, -0.01358024775981903, -1.2851972579956055, -0.6425012946128845, -0.8546665906906128, -0.2418638914823532, -1.2977627515792847, -0.47288766503334045, -1.0486383438110352, 2.3775527477264404, -0.47281506657600403, -1.0005865097045898, -0.19914498925209045, 1.1503938436508179, 1.1562656164169312, -0.592710554599762, 0.6356798410415649, 0.7639639377593994, 2.162557601928711, 0.25090211629867554, 0.49589672684669495, 0.25514039397239685, 0.10012692958116531, -0.8377197980880737, 1.04123854637146, 0.2515137791633606, -1.4107236862182617, 0.8196266293525696, -1.0013731718063354, 0.9711453914642334, -1.4648046493530273, -1.8537158966064453, -0.1531125009059906, -1.4751064777374268, -0.7176320552825928, -0.26649507880210876, -1.3466851711273193, 0.9047253131866455, 0.7162307500839233, 1.496289610862732, 0.7140330076217651, 1.3286386728286743, 2.0258827209472656, 0.5829535722732544, 0.03742828592658043, 1.5540372133255005, -0.8958070278167725, 0.0999317467212677, 1.2211071252822876, 0.19333921372890472, 0.221406951546669, -0.7312954664230347, -0.6144536733627319, 0.5815787315368652, -0.19079478085041046, 1.0659832954406738, 1.2535879611968994, -0.35563158988952637, -1.2173242568969727, 0.9424014687538147, -0.9287099838256836, -0.4532921314239502, -0.23651763796806335, -0.188335120677948, 0.9246083498001099, 0.7586050629615784, -1.6202292442321777, -0.07236923277378082, 0.5793004035949707, -0.5901724100112915, 2.1825175285339355, 1.1840031147003174, -0.19599705934524536, -1.315857172012329, -0.597083330154419, 0.4271717369556427, -1.4510794878005981, 0.4962437152862549, -0.8353779315948486, -1.0249146223068237, 0.0720304548740387, 0.03301375359296799, 0.4806309640407562, -2.319597005844116, -0.9808682203292847, 0.6407691836357117, -0.4868687391281128, -0.495723694562912, 1.1834461688995361, 0.7273659706115723, -0.7480759024620056, -0.9797601103782654, 0.11291646212339401, 1.337623119354248, 0.9608978629112244, -0.06862996518611908, 1.6584748029708862, -1.1700388193130493, -0.8545547127723694, -0.4405982494354248, -2.0102665424346924, -0.4227771461009979, 0.7215030193328857, -1.094749093055725, 1.3868576288223267, 0.031162049621343613, -0.05778483673930168, -1.1631282567977905, 0.26575997471809387, 0.22709429264068604, 0.15136530995368958, -0.7719050645828247, -0.6411904096603394, -0.38587865233421326, -0.8035785555839539, 0.6502602696418762, -0.09960697591304779, -0.22125336527824402, -0.05500270798802376, -0.6773292422294617, -1.1735118627548218, -0.9796655178070068, -0.47927790880203247, -0.5018318295478821, 1.7920149564743042, 0.5798967480659485, 2.13761568069458, 0.7076562643051147, 0.5715166926383972, -1.3755770921707153, 0.33974775671958923, -0.017309201881289482, 1.0472445487976074, -0.1019846573472023, -0.7654647827148438, -0.9677004814147949, -0.17468731105327606, 0.47099268436431885, -1.7234320640563965, 1.7219493389129639, -0.40175020694732666, -1.8229598999023438, -0.526796281337738, 0.7899437546730042, 0.9569993615150452, 1.0135931968688965, -0.009553483687341213, 0.6507916450500488, -0.24705202877521515, -0.6885988712310791, 0.37776994705200195, 1.1785213947296143, -1.5556498765945435, 0.6368643045425415, 0.844035267829895, 0.3871768116950989, 0.2824234068393707, 1.0011032819747925, 0.5801958441734314, 1.01339590549469, 2.355778694152832, -0.567046582698822, 1.08842933177948, 0.9144588112831116, -0.5537911057472229, -1.4950473308563232, 0.7990477681159973, -0.29856210947036743, -1.3831263780593872, 1.836685299873352, 1.3131394386291504, 0.5782801508903503, -1.21187162399292, -0.6996601819992065, -0.9624119997024536, 0.11230231821537018, -0.09890951216220856, 0.011385551653802395, 0.963272213935852, -1.408074975013733, -0.0676521584391594, -1.5196043252944946, 0.8917244076728821, 0.8216201066970825, -0.31916308403015137, -2.1721878051757812, -1.2136746644973755, -0.6457808613777161, -0.7974262833595276, -0.053331125527620316, 0.6249707341194153, -0.15152712166309357, -0.3518831133842468, 1.7762274742126465, -1.6900142431259155, -1.280355453491211, -0.7591031789779663, -0.7579610347747803, 0.3213506042957306, 0.13357987999916077, 0.067228764295578, -0.8817699551582336, 0.5947338342666626, -0.7260075211524963, 0.5279327630996704, 1.6184362173080444, -0.10997002571821213, -0.5729943513870239, 0.28881025314331055, -0.8848828673362732, 1.3375481367111206, 0.33247724175453186, 0.014632917009294033, -0.5610706210136414, 0.5175048112869263, 1.6540794372558594, -0.6464660167694092, 0.4053117632865906, -1.1794856786727905, 0.4691006541252136, 1.8509521484375, -1.0167008638381958, 1.0654789209365845, -0.9122570157051086, -1.3071471452713013, -0.7470515370368958, -2.210225820541382, 0.8701345920562744, 0.6522208452224731, 0.07338953763246536, -0.8572772741317749, 0.45829716324806213, 0.9962385892868042, 0.6110658645629883, 0.16171148419380188, -1.2916911840438843, -1.7959644794464111, -0.16042903065681458, 0.5189864039421082, 0.9411975145339966, -0.3711022436618805, 0.40953493118286133, 0.05291692540049553, -0.5945321321487427, -1.2223619222640991, -1.7375972270965576, 1.372442364692688, 0.3630000948905945, 0.6329944133758545, -0.8459370136260986, -0.10282076895236969, -0.4611662030220032, 0.9399393200874329, -1.1406066417694092, -0.19285766780376434, -0.8744421005249023, 0.7122641801834106, -0.14761458337306976, -0.34906166791915894, -0.02909902110695839, 0.2729730010032654, -1.0250673294067383, -2.0374600887298584, -1.417500615119934, 0.6937747001647949, -0.24453166127204895, -0.318521648645401, -1.6940970420837402, 1.152124047279358, 0.3096263110637665, 1.5500885248184204, 1.4541347026824951, 0.8445941209793091, -0.914495587348938, 0.8413196802139282, 1.3973174095153809, -0.7327566742897034, -1.4294899702072144, -0.15161505341529846, 1.4184798002243042, -0.937484860420227, -0.33268800377845764, -0.8925302624702454, -0.48438122868537903, 2.0500075817108154, 0.5349916219711304, -0.6846747398376465, 0.05404443293809891, 0.07718641310930252, -0.8476062417030334, 1.342940330505371, -1.8539162874221802, 1.7753651142120361, 0.018052242696285248, 0.7283474802970886, -1.0250848531723022, 0.6067435145378113, -0.06473706662654877, 0.9469010829925537, 1.1180633306503296, -0.4604300856590271, -0.04333324357867241, -1.2476563453674316, -1.3947904109954834, -1.9572707414627075, 1.1068086624145508, -1.6885948181152344, 0.14762963354587555, -0.3228036165237427, 0.3176230788230896, -0.2158348113298416, -0.6698979735374451, 2.4157490730285645, -0.7052255868911743, -1.5924088954925537, -0.5827881693840027, -0.36647334694862366, -0.02386264130473137, 2.1155803203582764, -0.6105407476425171, -1.442461609840393, -1.0925407409667969, -1.6846760511398315, 2.7626383304595947, -0.3902706205844879, 0.6880801320075989, -0.6984928846359253, 2.024336576461792, -0.03310665115714073, -0.004224221687763929, 2.1947193145751953, 0.7467390894889832, -0.009381630457937717, 0.9242751598358154, -1.0106074810028076, 0.5288928747177124, 0.5061697959899902, 1.2106317281723022, -0.46499255299568176, 0.40139973163604736, -2.2209279537200928, 0.06780439615249634, 0.12240701913833618, -0.05108644440770149, -0.5504409670829773, 0.06602483987808228, 0.21243314445018768, 0.0902942419052124, 0.6354888081550598, -1.1727079153060913, -0.5005060434341431, 1.5728380680084229, 0.019963787868618965, 1.17868173122406, 0.6324576139450073, 0.5469000935554504, -0.5186207890510559, 0.17832469940185547, -3.2702314853668213, -0.6124941110610962, 0.6496748328208923, -0.4430176615715027, -0.5607913136482239, 0.5692419409751892, -0.15969625115394592, -0.2334473580121994, 0.5711982846260071, 0.64302659034729, -1.7755498886108398, 0.578956127166748, -2.1717894077301025, 1.7836991548538208, 2.3527824878692627, -0.22550968825817108, -1.5482468605041504, -0.660070538520813, 0.65847247838974, 0.7821130156517029, 0.8611628413200378, -0.3889106512069702, 0.5727222561836243, -0.8095758557319641, 0.6286453008651733, 1.7246379852294922, 1.7185167074203491, 1.2567319869995117, 1.7254291772842407, 0.08683803677558899, -0.10170885920524597, 1.4525315761566162, 0.7326322197914124, 0.8945884704589844, -1.0304960012435913, 0.5706346035003662, 1.4213517904281616, -0.01183589082211256, 0.2650916576385498, 0.4342676103115082, -0.232412651181221, -0.7872820496559143, -0.039569880813360214, 0.8679186105728149, 0.5219478011131287, 0.6182418465614319, 0.3292303681373596, 0.5575428605079651, 0.7205345034599304, -0.06809020787477493, -0.7158960700035095, 0.9952023029327393, -0.27409082651138306, -0.6284710168838501, 0.8282082080841064, -2.0909862518310547, -1.543451189994812, -0.8555532097816467, -0.8632721900939941, -0.07838872075080872, -0.574808657169342, -1.356282114982605, 0.34341666102409363, 3.148754835128784, -0.881808876991272, -0.638066828250885, -0.5826100707054138, 0.7109254598617554, -0.7902633547782898, -1.6785993576049805, -0.636659562587738, 0.22988152503967285, 0.8347785472869873, 1.0640722513198853, 0.9116054773330688, -0.31560155749320984, 0.9582749605178833, 0.4331318736076355, -1.240920901298523, 0.3904343247413635, -0.031030641868710518, 0.06249747425317764, 1.716564655303955, 0.07132159918546677, 2.277501344680786, 2.0159313678741455, -0.6210445165634155, -2.4710700511932373, -0.11522436141967773, 0.5416016578674316, 0.5578361749649048, 0.5720890760421753, -0.08702252060174942, -0.017865240573883057, 0.735220193862915, -1.2770487070083618, 0.9100050926208496, -0.12196199595928192, -0.4230569005012512, 1.0357270240783691, -0.7100521922111511, 0.950589120388031, 0.83664470911026, -0.24786993861198425, 0.040006160736083984, 0.6800481677055359, -1.057106375694275, 1.4119822978973389, -1.1512056589126587, -0.8517401218414307, 1.5981608629226685, 0.575089693069458, 0.38556456565856934, 1.179581642150879, 0.5727436542510986, 0.03439422696828842, 0.18475574254989624, 0.10573583096265793, 0.32292449474334717, -0.4094414710998535, 0.6854500770568848, -1.3881735801696777, 0.838159441947937, 0.17855678498744965, 0.24314343929290771, -0.4157579839229584, -0.24827811121940613, 1.7031573057174683, -0.5634682178497314, -0.06896170228719711, 0.9076427221298218, -1.0802619457244873, -1.1508674621582031, -0.5760336518287659, -0.8314602375030518, 0.7606425285339355, -0.9757012724876404, 0.7230045795440674, -1.5241193771362305, 0.8046414852142334, 0.10746694356203079, -0.03408403694629669, -1.1782631874084473, 0.7448094487190247, 1.3675309419631958, -1.0514349937438965, 0.4368978440761566, -0.773432731628418, -0.40439358353614807, -0.060129985213279724, -0.22477838397026062, -0.5245327353477478, 0.802747905254364, -0.2986260652542114, 0.5569589138031006, 0.2872450649738312, 0.6485764980316162, -1.7440896034240723, -0.33886778354644775, 1.4863216876983643, -0.32680222392082214, -0.8203592896461487, -0.5118633508682251, 0.07968634366989136, -0.5336511731147766, -0.22317807376384735, -0.033277515321969986, 0.018833667039871216, 0.030495531857013702, -1.291473388671875, -0.390313982963562, 2.8545901775360107, 0.29272469878196716, -1.686514973640442, 1.1863958835601807, -1.6890925168991089, 0.26162177324295044, 0.4631012976169586, 1.5633914470672607, 2.552476644515991, -1.7841181755065918, 0.14517228305339813, -0.9494175314903259, -0.11814585328102112, 1.134430170059204, 0.7053744196891785, 0.7604461312294006, -1.0010528564453125, 1.2595447301864624, 0.43260136246681213, -0.2763282358646393, -1.764997959136963, 0.7815316319465637, -1.124411702156067, -0.8619887828826904, -0.7460647225379944, -0.9415415525436401, -0.2027035653591156, 0.8350843191146851, -0.3138701021671295, 0.7777580618858337, -0.8332269787788391, 0.6017027497291565, 0.5370591878890991, -2.1874871253967285, 1.1574586629867554, -1.1286758184432983, 0.6845775246620178, -1.3837534189224243, 0.46193620562553406, 0.1662420928478241, -0.15009130537509918, -1.316579818725586, 0.0869290828704834, 0.787135899066925, -0.8368744254112244, 0.04379808530211449, 0.16878187656402588, 0.7720447182655334, 0.6076355576515198, -1.2245635986328125, 0.2900588810443878, -0.5184334516525269, -0.18413467705249786, 1.2013534307479858, 0.6851714849472046, 0.13588427007198334, -0.5486359596252441, 2.4700841903686523, -1.4903467893600464, -1.287777066230774, -0.33183935284614563, -0.022446099668741226, 0.37740492820739746, 0.8895962834358215, -1.0922284126281738, -0.3867678642272949, -0.9552155137062073, -0.09908173978328705, 0.9465484619140625, -0.8803337812423706, 0.9877886176109314, -0.6713119745254517, -0.6456983685493469, -0.5644642114639282, -0.48181164264678955, 0.4636537432670593, -0.6260489821434021, 0.7817568778991699, 0.6303743720054626, 2.01692795753479, -0.5283945202827454, 0.3353220522403717, -0.5666877627372742, -1.4220163822174072, 1.2082083225250244, -1.183907389640808, 1.4515520334243774, -2.1398355960845947, -0.928036093711853, -1.1800287961959839, -0.059865888208150864, 1.0237396955490112, -1.016622543334961, 0.3408981263637543, 1.4685980081558228, 0.7485753893852234, -2.1101279258728027, 0.8738845586776733, -1.9235599040985107, -1.1292933225631714, -1.6334831714630127, 1.3442742824554443, -0.3224489986896515, 0.05344941467046738, -0.3327066898345947, -1.0023972988128662, 0.3326459527015686, -0.6769485473632812, -0.48590925335884094, 0.748650074005127, 0.7254233956336975, -1.3926255702972412, -0.144434854388237, -0.13298898935317993, -0.3003867566585541, -0.4070984721183777, -0.21263691782951355, -0.4608244299888611, 0.49688273668289185, -2.501218318939209, 0.6468271613121033, -0.2894287109375, -1.2799031734466553, -0.14132097363471985, 0.5887300372123718, -0.5825024247169495, 0.22763416171073914, -0.6694647669792175, 0.7773107290267944, 1.1098997592926025, 1.1270875930786133, -1.2189642190933228, 0.4970752000808716, -1.0696481466293335, 0.7976208925247192, -0.7646276950836182, -0.6986985802650452, 1.2135207653045654, 0.644945502281189, -0.5110988616943359, 0.18740852177143097, -1.203903079032898, -2.2012627124786377, -1.0831575393676758, 2.3270795345306396, -0.03865952417254448, 0.6277605295181274, 1.2354323863983154, 0.301899790763855, 0.08934173732995987, -0.9334846138954163, -1.0584992170333862, -0.5729190707206726, 0.7341500520706177, -0.7144914865493774, 0.4331783056259155, 0.14284023642539978, -0.3205089867115021, 0.6724944710731506, -0.9152591824531555, -0.5903190970420837, 0.28959277272224426, 2.3069875240325928, 0.7813430428504944, -0.0716656818985939, -1.893208384513855, -0.8656192421913147, 0.43031764030456543, -1.1614943742752075, -0.42000988125801086, 0.22294606268405914, 1.53873610496521, 0.25309401750564575, -0.3042459487915039, -1.4163202047348022, 1.5688045024871826, -0.5005454421043396, 0.46101516485214233, 0.651035726070404, 1.057582974433899, 0.048908885568380356, 1.2667515277862549, -1.69040846824646, 0.16726036369800568, 0.6573064923286438, -0.9846068620681763, -0.6088714599609375, -0.6465652585029602, 0.1344841867685318, 1.2209105491638184, 0.5980466604232788, 0.6224650144577026, -0.2673308253288269, -0.4354613423347473, 0.8210893869400024, 0.248748317360878, 0.04525880888104439, 0.6470990180969238, 0.3358505964279175, 1.0593088865280151, -0.2547401487827301, -0.26539358496665955, 0.47269731760025024, 2.6671431064605713, -0.5371990203857422, 0.8755315542221069, -1.0691523551940918, 0.13743728399276733, -0.4871867299079895, 2.0908408164978027, 2.543889284133911, 0.14864344894886017, -0.23863659799098969, -0.43803393840789795, -0.5212417840957642, 0.20046189427375793, -0.6947271227836609, -0.20614446699619293, 0.9254767894744873, 0.7091622352600098, -0.28769567608833313, 1.3055318593978882, 0.7548636198043823, 0.858511745929718, -1.3086581230163574, 1.1413620710372925, -1.175876259803772, -1.37185537815094, -0.7745978832244873, 0.36082467436790466, 1.191552758216858, -1.0100231170654297, -0.22489170730113983, -0.26156938076019287, -0.28084343671798706, -1.4342570304870605, 0.4374183416366577, -1.9989453554153442, -1.197417974472046, -2.171290636062622, 1.398656964302063, -0.6181240677833557, 1.4626282453536987, 2.7558844089508057, 1.3777281045913696, 1.2808887958526611, 0.6553411483764648, -0.24432605504989624, -1.1007763147354126, -0.6562913060188293, 0.25738486647605896, -1.0623085498809814, 0.3320506513118744, -0.8560898303985596, -0.12626174092292786, -1.4205933809280396, 2.1684720516204834, 0.487698495388031, 1.8475341796875, 0.33836132287979126, -0.7392135262489319, -1.5152392387390137, 0.6104001402854919, 1.0408015251159668, 0.4282659590244293, 1.5804849863052368, -0.9259535074234009, -0.46124276518821716, 0.4192890524864197, 0.03977634385228157, 1.5626198053359985, -0.5631765723228455, 0.8668378591537476, 0.3236064314842224, 0.73525470495224, 0.3524424135684967, -1.3182265758514404, -0.8281418085098267, 0.5210743546485901, -0.5308452844619751, 0.8239394426345825, -1.971261739730835, 1.5339239835739136, -0.2054385393857956, -1.0975698232650757, -1.4689686298370361, -1.0512853860855103], [-0.34926196932792664, 0.11755980551242828, 1.7598508596420288, -1.3295613527297974, 0.5306268334388733, 0.11881402134895325, -1.4636417627334595, 0.34329065680503845, -0.1227654367685318, 0.7703374624252319, -0.7429064512252808, -0.9236295223236084, -2.8661301136016846, 2.647435426712036, 1.192312240600586, 0.6653568148612976, -1.6174851655960083, -0.6318703293800354, 0.6725719571113586, 0.07807537168264389, -0.7568768858909607, 1.3779373168945312, 1.077082872390747, -0.9109940528869629, 2.0307796001434326, -1.037864089012146, 1.8292350769042969, -0.5380024909973145, -0.8672420382499695, 0.9668857455253601, -2.377627372741699, 0.0038399596232920885, -0.490407794713974, -0.21824312210083008, -0.6790302395820618, 1.1115111112594604, 1.1528048515319824, 1.7596575021743774, -0.4633656442165375, -0.0747232586145401, 0.3515525758266449, -0.3251478374004364, -0.2325216382741928, -0.6969648599624634, -0.7970761656761169, 1.9555352926254272, 1.2235381603240967, -1.7464909553527832, 0.09047878533601761, -0.06099222972989082, 0.49830928444862366, -0.33148616552352905, 0.043655913323163986, 0.6311070322990417, -2.362755298614502, -1.7403461933135986, 1.4741688966751099, 1.1051968336105347, 0.006328577175736427, 0.30474454164505005, 1.1536085605621338, -0.32281190156936646, -1.1278890371322632, -0.1718619167804718, 2.3088629245758057, 0.27302008867263794, -1.0163021087646484, 0.05123603343963623, 1.8323397636413574, -0.1922920197248459, 1.447134017944336, -0.557982325553894, -0.8140850067138672, 0.3059574067592621, -0.2422722727060318, 0.6162413358688354, -0.0002582839806564152, 0.4084216356277466, 0.30401045083999634, -0.22637267410755157, 0.22674335539340973, -0.06864215433597565, -0.26244547963142395, 0.9954977631568909, -1.5640709400177002, 0.46332934498786926, 3.184455633163452, -3.1325128078460693, -0.015941651538014412, -0.17046789824962616, 1.639352560043335, -0.6518676280975342, -0.20449553430080414, -2.1574435234069824, -1.8148638010025024, 0.28585195541381836, 0.31038740277290344, 0.08831533789634705, -0.23755963146686554, -0.6636858582496643, 0.6499692797660828, 0.02069246955215931, -0.4742983877658844, -0.8369698524475098, 2.4688968658447266, -0.0914001613855362, 2.1066434383392334, 0.13051481544971466, 0.8584364652633667, -0.12842506170272827, -0.05336356908082962, -1.344622015953064, -0.9131849408149719, 0.4049687385559082, 0.3758692145347595, 2.440035343170166, 1.355157732963562, 0.3239366412162781, -0.023669304326176643, -1.071765661239624, -1.4087318181991577, -2.633608102798462, -0.27426236867904663, 0.2753666937351227, -1.9039885997772217, -0.8092461824417114, -0.511342465877533, -0.3094976246356964, 2.2647876739501953, 1.0291849374771118, 1.2217625379562378, -0.5156863927841187, 0.37666332721710205, 0.7738367915153503, -0.9248249530792236, 0.7891847491264343, -0.6410344839096069, -0.8236694931983948, -0.2975665330886841, 0.9514540433883667, 1.07636296749115, 1.4910216331481934, 1.341665506362915, 1.2339744567871094, 0.853684663772583, 0.9428288340568542, 1.0727871656417847, -0.194771870970726, -0.7380759716033936, 0.03639284893870354, 1.6315717697143555, -0.08389490097761154, -0.13327458500862122, 0.30839887261390686, -0.4988252520561218, 0.41407251358032227, 0.35278648138046265, 0.1377432644367218, -1.7020169496536255, -0.7368550300598145, 1.8815473318099976, 1.760297417640686, -0.9346529841423035, 0.40055787563323975, -0.14599564671516418, -0.20380331575870514, -0.4965919554233551, -0.4489571750164032, 0.28250613808631897, -0.8003730177879333, 1.5269203186035156, 0.5269826054573059, 0.7495949268341064, 0.6054326295852661, -1.5767254829406738, -0.08874667435884476, -0.005332328379154205, -0.8732010722160339, -1.0655919313430786, 1.474404215812683, -0.984769880771637, -0.9937465786933899, -0.00020141017739661038, -0.582106351852417, 2.4050886631011963, -0.8746525645256042, 0.9991808533668518, -0.19176173210144043, -0.8324146866798401, -1.408542513847351, -0.4964592456817627, 0.4732878804206848, -1.0434606075286865, 0.08363626152276993, -1.2566776275634766, 1.6776942014694214, -0.015692545101046562, 0.7944296002388, 0.22264739871025085, -2.9833340644836426, 0.6145041584968567, -0.19644202291965485, 0.23476015031337738, -1.0284526348114014, -0.6601639986038208, -0.3511076271533966, 0.8879894018173218, 2.04154634475708, -1.1793711185455322, 0.40481147170066833, 2.25825572013855, -0.6337764859199524, -1.1444553136825562, -0.5714449286460876, 0.7081159353256226, 2.1587717533111572, 1.11892569065094, -1.1154046058654785, 1.9023021459579468, 0.6980894207954407, -0.809942901134491, 0.0009819550905376673, 0.23133748769760132, -1.8958407640457153, 0.7097637057304382, 0.13159435987472534, 0.7454558610916138, 1.4364371299743652, -1.9759376049041748, 0.17531952261924744, 0.3569105267524719, -0.24203288555145264, -1.373908519744873, 2.0296337604522705, 1.9474149942398071, 0.4186003506183624, -1.5538852214813232, -1.245456576347351, -0.10465158522129059, -1.1687312126159668, -1.4860337972640991, -1.092056155204773, 1.9042503833770752, 0.46747151017189026, 0.8137670755386353, -1.7811226844787598, -0.7638456225395203, -1.8853243589401245, -1.0446211099624634, -0.7103832364082336, 0.8700447678565979, 0.4403354823589325, -0.10958623141050339, 0.3403942584991455, -1.1130704879760742, 0.5853110551834106, 0.18136076629161835, -1.0403696298599243, -1.397092580795288, 0.16795721650123596, -0.5894450545310974, 0.13966037333011627, -0.49676525592803955, -0.053436268121004105, -0.40865206718444824, 0.39084523916244507, -0.7099647521972656, -0.34491169452667236, 0.8661956787109375, 0.38322174549102783, 1.1094638109207153, -0.7837424278259277, -0.7857321500778198, 0.7848034501075745, -0.5868671536445618, 0.06755660474300385, 0.03832481801509857, -1.058796763420105, -0.8453067541122437, -0.7549819946289062, 1.6073153018951416, 1.5412570238113403, -0.39360424876213074, -1.1383007764816284, -0.4772520661354065, -1.03811514377594, 0.4614317715167999, -0.09898851811885834, 0.7892636656761169, 0.9058142900466919, 0.046665582805871964, -0.3102203607559204, -0.45461538434028625, -0.9203376770019531, -0.2771550714969635, 1.027772068977356, -2.4473345279693604, 0.6687767505645752, 1.137471318244934, 1.8861517906188965, 0.5513617396354675, 0.31479451060295105, -1.0365571975708008, -1.2298130989074707, 0.7958340644836426, 1.48563814163208, -0.44721099734306335, 0.20132221281528473, -0.048287808895111084, -0.011168278753757477, -0.7914518713951111, -0.5888136029243469, -0.12271816283464432, -1.309502124786377, -0.3445129990577698, 0.15185005962848663, -0.4296411871910095, 0.3305893838405609, -0.17912925779819489, -0.6037765145301819, -0.5474865436553955, 1.3483809232711792, 1.4735972881317139, 0.9808924794197083, -0.6228113770484924, 0.9886294603347778, -1.1230090856552124, -1.354617714881897, -0.6355078816413879, -0.17105183005332947, -0.5222312211990356, -0.7457291483879089, 0.14809201657772064, -2.0021564960479736, -0.05760066956281662, -1.513311743736267, 0.38501110672950745, 0.34144988656044006, 1.6816171407699585, -0.6850904822349548, -0.5995299816131592, -1.592992901802063, 0.02041902206838131, 0.4185998737812042, 0.8209227919578552, 0.24838125705718994, 1.0743299722671509, -2.5830399990081787, 1.1567883491516113, 0.5136253833770752, 0.3532438278198242, -0.11242910474538803, -0.17990459501743317, -1.933474063873291, 1.2439837455749512, 1.7736515998840332, 1.7354990243911743, 0.5210290551185608, -0.8212497234344482, -0.4282209575176239, 1.2144310474395752, -0.9304388165473938, -0.25849875807762146, -0.5947191119194031, 0.6323575973510742, -0.6936450004577637, -1.219782829284668, -0.2920569181442261, 0.08395389467477798, -0.6570066213607788, 0.7244628667831421, -1.5235494375228882, -1.676777720451355, -0.414725661277771, 0.6645062565803528, -2.449721097946167, -0.7150960564613342, -2.5452444553375244, 2.5628578662872314, 0.3944454789161682, -1.5341777801513672, 0.3514164984226227, -0.21008972823619843, 0.7600395083427429, -0.5804609656333923, -0.38181883096694946, 0.8349005579948425, 1.9787089824676514, -0.3306857645511627, 0.9870193004608154, 0.11327128857374191, -0.06879157572984695, -0.13166210055351257, -0.3983401954174042, -2.379779100418091, -0.5824165344238281, 0.10694757103919983, 0.5079704523086548, 0.555406391620636, 1.4661256074905396, 0.8981736898422241, -0.22950029373168945, 1.9097466468811035, -0.07702415436506271, 0.9164313673973083, -0.11919394135475159, 1.1326271295547485, -1.847303032875061, 0.10171715170145035, -0.19613732397556305, 0.9403340220451355, 1.085264801979065, 0.26416119933128357, 0.08418958634138107, -0.33229154348373413, -0.027769191190600395, 0.32442018389701843, -1.535712718963623, -0.2884405255317688, 2.0372657775878906, 0.14143148064613342, -0.25577494502067566, -0.11603251844644547, 0.013336306437849998, -0.9847544431686401, -1.2602735757827759, 0.10946246236562729, 1.9624338150024414, 1.0709490776062012, 0.8453341722488403, -1.0694338083267212, -0.27491992712020874, -0.7513195872306824, 0.0461263433098793, 1.5123804807662964, -1.726637840270996, 0.36881473660469055, -0.09433694183826447, 1.7804096937179565, 0.18170848488807678, 0.9167401194572449, 0.7583772540092468, 0.13984934985637665, 1.0952470302581787, -0.4732726514339447, -0.4699324667453766, 1.1525886058807373, 0.16133777797222137, 1.578629732131958, -0.8499829173088074, 0.5889658331871033, 0.0070398664101958275, 1.1049823760986328, 0.956284761428833, 0.07730446010828018, -0.1253276914358139, -2.0199925899505615, 1.6155354976654053, -0.5801228284835815, 0.24757637083530426, -0.3978863060474396, 0.10312710702419281, 0.7595491409301758, -0.23387612402439117, -0.6025122404098511, 1.8573874235153198, 0.2518220841884613, 0.48198801279067993, -1.3741759061813354, -0.05417676642537117, -1.6938742399215698, 0.28715574741363525, -1.449531078338623, 0.16609500348567963, -0.972221851348877, -1.1983394622802734, -0.964531660079956, -0.8194448947906494, 1.6516602039337158, 0.7902162671089172, 1.5442509651184082, -2.523207187652588, 0.9781774878501892, 0.6769368648529053, 0.7082046270370483, 0.21441464126110077, -0.67267906665802, 1.6118900775909424, -0.061761729419231415, -0.22532470524311066, -0.9838421940803528, -0.15879607200622559, -0.7476446628570557, -1.1050419807434082, -2.6240880489349365, -0.25035929679870605, -0.8901997804641724, -0.8943648934364319, 1.3562694787979126, 0.7884268164634705, -0.05168231576681137, 1.523589015007019, 2.6171207427978516, 1.5859754085540771, 0.8462979793548584, 0.6682006120681763, 0.6612972021102905, -0.3420886993408203, -1.1515929698944092, -0.17814064025878906, -0.6630876660346985, 0.10927727073431015, -0.9328172206878662, 1.2482775449752808, -0.20242200791835785, 0.1760520488023758, 1.2888294458389282, -0.46565476059913635, -1.0141452550888062, 0.35810741782188416, -1.3259392976760864, 0.716988205909729, -0.6195462346076965, -0.10902427136898041, 1.771927833557129, -0.4463898837566376, -2.3593192100524902, -0.008372709155082703, -0.6037822961807251, 0.3427959680557251, 0.9114084243774414, 0.023876138031482697, 0.047001566737890244, 0.10078004747629166, -0.164316326379776, 0.5510990619659424, -0.10254014283418655, 0.15493707358837128, 0.3076428174972534, 1.5759673118591309, -0.9536272883415222, -0.831555187702179, -0.13096804916858673, -0.17205367982387543, 0.8875247836112976, -0.36444634199142456, -0.49956899881362915, -0.006636751815676689, -0.6636351346969604, -1.2430665493011475, -1.062155842781067, 1.699012041091919, 2.3942484855651855, -0.14164327085018158, -0.30017754435539246, 1.1229194402694702, 0.1642419546842575, -0.2881203293800354, 1.626213550567627, -1.7668728828430176, -0.5141091346740723, 0.17872512340545654, 0.3434716761112213, -0.10625779628753662, -0.9588633179664612, -0.9598856568336487, -0.02859114669263363, 0.9188621640205383, 0.9235864877700806, -0.782373309135437, -0.8062207102775574, 0.06915556639432907, 0.18602192401885986, -0.8902466297149658, 0.17268362641334534, 0.26538610458374023, -0.957311749458313, -0.060749780386686325, 0.07310444116592407, -0.0579579696059227, -1.2652902603149414, 0.10573354363441467, -0.4935500919818878, 0.10567506402730942, 0.37935709953308105, -0.605044960975647, 0.04178149998188019, 0.29270023107528687, -0.008073720149695873, 1.9855148792266846, 2.001861572265625, 0.045762721449136734, 0.4205581545829773, 0.9714009761810303, -0.5041722655296326, -1.045655608177185, -0.7660361528396606, 0.6788840889930725, -1.0537971258163452, -0.6397340893745422, 0.8367777466773987, -0.6156107187271118, -0.5312853455543518, -0.9590609073638916, -0.7859566807746887, -1.2952555418014526, 0.6446076035499573, 0.33200961351394653, -0.8297932744026184, 1.1270357370376587, 1.1700870990753174, 1.5087957382202148, 1.6610842943191528, -0.31738027930259705, 1.8533358573913574, -1.0893499851226807, 0.8943763971328735, -1.0627058744430542, -0.5805975794792175, -0.5577418804168701, 0.13492707908153534, 0.07643839716911316, -1.0654947757720947, -0.12637121975421906, -0.25699564814567566, -1.265498399734497, -0.9467695951461792, -1.022361159324646, -1.477519154548645, -0.9859493970870972, 0.7494535446166992, -0.21352989971637726, 0.7805179357528687, -1.0626654624938965, 0.6675720810890198, 1.935707926750183, -1.3252482414245605, 0.8309276103973389, -0.4462854564189911, -0.5108713507652283, 1.3338977098464966, -0.7339878678321838, -1.8116050958633423, 2.3257315158843994, 1.600987434387207, -1.1179529428482056, 1.064797043800354, 1.2317677736282349, -0.5486151576042175, 0.5549476146697998, 1.8469069004058838, -0.26309412717819214, 1.0334160327911377, 0.37776267528533936, -1.8310871124267578, -0.44165727496147156, 0.4424043297767639, -0.8267194032669067, -0.9094483256340027, 0.6859546303749084, -1.3534908294677734, 1.2038123607635498, 1.2026821374893188, 1.2626153230667114, 0.6606401205062866, -0.718540608882904, -0.2358980029821396, -0.7958101630210876, -0.804656445980072, 1.1633086204528809, -0.12773625552654266, -0.2732912003993988, 0.4851315915584564, -1.4478224515914917, -0.6695601940155029, -0.566703736782074, 0.06132656708359718, 0.46745792031288147, 0.8340857028961182, 0.7929316163063049, 1.3765966892242432, -0.33053192496299744, 1.179866909980774, -0.023881077766418457, -1.1462117433547974, -0.5324953198432922, -0.1455431878566742, 0.3948269784450531, 0.7797816395759583, -0.1001502275466919, 0.4696800112724304, -0.030797041952610016, 0.45049819350242615, -0.4788438677787781, 0.9760568141937256, -1.149008870124817, 0.4631046950817108, -0.7875024676322937, 0.30414000153541565, 0.7594295740127563, 1.1935079097747803, -0.7789605855941772, -0.12732906639575958, -0.36285117268562317, -1.3086446523666382, 0.6065875887870789, -0.8310093879699707, 0.8475529551506042, -0.38530033826828003, -1.1208804845809937, 1.6247899532318115, -1.2640780210494995, -0.8264720439910889, -2.059511423110962, 1.195082664489746, -0.1130293607711792, 0.6186617612838745, -2.043936252593994, 1.7086760997772217, 0.2530632019042969, 0.611225426197052, -1.2002618312835693, 1.72110116481781, 2.975787878036499, 0.7216464877128601, -0.0872325673699379, -1.3725072145462036, -1.3472939729690552, -1.3698784112930298, 0.4735826849937439, 1.9712518453598022, 1.0224459171295166, -0.8664047122001648, -0.3683329224586487, 0.8517112135887146, -0.7299385070800781, 1.3504431247711182, 1.3156216144561768, -0.39724448323249817, -0.27434512972831726, -0.5912928581237793, 0.8473827838897705, 1.4070159196853638, -0.7274413704872131, 0.49330076575279236, 0.0018862043507397175, -0.47212591767311096, 0.5971693396568298, -0.11278438568115234, 0.23191384971141815, -0.7536264657974243, 0.3635917007923126, -0.7626762986183167, -0.7793669700622559, 0.5429795980453491, -0.6846389174461365, -0.06804818660020828, -0.9973116517066956, -0.02979906089603901, -0.2211260199546814, 0.353731244802475, -0.7196774482727051, -1.0114973783493042, 1.4890720844268799, -0.07089874893426895, 0.820878267288208, -0.08458072692155838, -1.6402982473373413, -1.4172990322113037, -0.52533358335495, -0.500012218952179, -0.8219273090362549, -0.7993674874305725, -0.14953744411468506, 1.2697147130966187, -1.731186032295227, 0.38940292596817017, 1.617135763168335, -2.008061408996582, 1.1153119802474976, 0.668865442276001, -0.22664114832878113, -1.3357021808624268, 0.2712171673774719, 0.988106369972229, 0.24598947167396545, -0.13218888640403748, -0.22187870740890503, 1.2357474565505981, -0.21836301684379578, -0.3585057854652405, 0.4279446005821228, -0.7248676419258118, 1.2312476634979248, -0.7737694978713989, -0.16867183148860931, 2.433448076248169, 0.0012157764285802841, -0.8575605154037476, -0.3959029018878937, 1.4798911809921265, 1.743894338607788, 0.6971347332000732, 2.0372154712677, -0.4320732057094574, 0.9268984198570251, -0.30816420912742615, -0.5959587097167969, 0.0350901260972023, -0.45466724038124084, 0.1504574716091156, 2.7648093700408936, 1.3966833353042603, 0.012056599371135235, 0.585392415523529, -1.3457262516021729, -0.6484744548797607, -0.9352226257324219, 0.2905265986919403, 0.5083097219467163, -0.41437292098999023, -0.9854305982589722, 0.25514858961105347, -0.7439289093017578, -0.38154879212379456, 0.3574974834918976, 0.07233569025993347, -0.5189221501350403, 1.3728188276290894, -0.9836352467536926, -0.16145005822181702, 0.09673924744129181, 1.7713114023208618, 0.6635091304779053, 0.7906224727630615, 1.585477352142334, -0.1460944414138794, -1.1781952381134033, -0.12177158892154694, 0.4600376784801483, -0.8125109672546387, -1.950927972793579, 1.0487585067749023, -0.4227316975593567, 1.9819581508636475, 0.5289545655250549, 0.03032960183918476, -2.089371919631958, -0.5063819289207458, -0.7021760940551758, -1.7378358840942383, -0.7800403833389282, -1.630537748336792, -2.03279447555542, -1.3715635538101196, -0.8114588856697083, -0.4826441705226898, 0.5780758857727051, 0.7800779938697815, 0.2553866505622864, -1.0362955331802368, 0.7052595615386963, -1.4239461421966553, -0.9137250185012817, 0.13689245283603668, -0.0514102578163147, 0.1302243322134018, 1.0634839534759521, 1.808770775794983, 0.6818404793739319, -0.6783322691917419, -0.5147621631622314, -0.7587312459945679, -1.7077598571777344, -0.40004000067710876, 0.4333239197731018, -0.05968114361166954, 1.3789241313934326, -0.40265733003616333, -0.7963412404060364, -0.819986879825592, 0.33552080392837524, -0.569537878036499, -0.9791803956031799, -1.6905412673950195, -1.1772818565368652, -0.49199360609054565, 0.29747724533081055, -0.5060028433799744, -1.0557643175125122, 0.6325779557228088, -0.6651300191879272, -0.7148210406303406, 1.0064245462417603, -0.6447513699531555, 0.3660813868045807, 0.6038991808891296, -0.8750340342521667, 1.2847076654434204, 0.20718124508857727, -0.8333042860031128, 0.20062510669231415, -0.5017082691192627, -0.7716913819313049, -0.07671040296554565, 0.676491379737854, -0.5553907752037048, 0.7077307105064392, 0.3304043412208557, -0.7954662442207336, 0.46121111512184143, -0.16195616126060486, -0.5923465490341187, 0.21396078169345856, 2.5421791076660156, 0.5808088779449463, 0.08217203617095947, 0.816199779510498, 0.43537360429763794, -0.5365535020828247, -0.35475146770477295, 0.37354227900505066, -0.5579133629798889, -0.6368760466575623, 2.7334606647491455, -0.08877876400947571, 0.07089067250490189, 0.1981106698513031, -0.5427541136741638, -1.449689269065857, -0.9614775776863098, 0.9285954833030701, -1.3754383325576782, 0.9708019495010376, 0.17488929629325867, -0.23497319221496582, 0.4406285881996155, 0.6022423505783081, 1.108989953994751, 0.7575005888938904, -1.301941990852356, -0.23382540047168732, 0.011705068871378899, 0.19689878821372986, 0.30274781584739685, 0.9311768412590027, 0.0831054151058197, -0.0575731061398983, -0.8828652501106262, -0.37668880820274353, 0.782736599445343, 1.6168906688690186, -0.18234150111675262, -0.14802561700344086, 0.6003955006599426, -0.21037878096103668, 0.9556963443756104, -1.337169885635376, -1.5266847610473633, 0.5609544515609741, 0.12172196060419083, 0.705799400806427, -1.7570266723632812, 1.3100864887237549, -1.397165060043335, 1.907591700553894, 0.6897816061973572, 0.7853986620903015, 0.9388580918312073, 0.36457863450050354, 0.3105989694595337, -0.5002613067626953, 0.0658414363861084, -1.427633285522461, 0.5259929299354553, -0.5473506450653076, -0.7154523134231567, -0.5770902037620544, 0.3355589210987091, -0.16102397441864014, -1.0793037414550781, 0.1839374303817749, -0.4063710570335388, 2.0775914192199707, 0.1118098571896553, -1.0270289182662964, 1.5082063674926758, 0.31282052397727966, -0.15171010792255402, -0.8142539262771606, -0.27150478959083557, -0.20223668217658997, 0.11568234115839005, -1.1502606868743896, 0.8678218722343445, -0.19365684688091278, 0.2976602613925934, 2.0484843254089355, 0.9763380289077759, -1.265170693397522, -1.3652081489562988, 0.4211846888065338, -1.1152653694152832, 0.2210106998682022, -0.5047917366027832, -1.6354273557662964, 0.5868127942085266, 1.370030164718628, 0.49497145414352417, -0.47788605093955994, 0.7643393278121948, -0.41570809483528137, 1.6443915367126465, 0.49306121468544006, -1.2247180938720703, 0.7449560761451721, -0.1937752664089203, 0.43883946537971497, -1.1404988765716553, 0.18639914691448212, -1.923058271408081, 0.47440236806869507, 2.565251588821411, -1.5979582071304321], [0.09354721009731293, 0.5140332579612732, -1.4860632419586182, -0.7271988391876221, -0.3834017217159271, 0.3040701746940613, -1.4440687894821167, -0.17418895661830902, 1.6358628273010254, -0.1511247158050537, 0.523820698261261, -0.2007405310869217, 1.0336769819259644, -0.15947122871875763, 0.7474316954612732, -0.5595976114273071, -0.5670804381370544, -1.6670892238616943, 0.8979994058609009, -0.35399898886680603, 0.8965288996696472, -0.3519494831562042, -1.7524785995483398, -1.4710142612457275, 0.912121593952179, -0.6735830903053284, -0.7149797081947327, -0.55323725938797, -0.8286415338516235, 1.0086179971694946, 0.026037568226456642, -0.9098740816116333, -1.3995850086212158, -2.1520588397979736, 0.11090338230133057, 1.1141383647918701, 1.2818857431411743, -0.05259546637535095, -1.0905437469482422, -0.9721975922584534, 1.2931641340255737, -0.3509802520275116, -0.543842613697052, 1.7929232120513916, 0.057940155267715454, 0.4275370240211487, 1.1720722913742065, 0.17370548844337463, -0.7695648670196533, 0.7471705675125122, 0.31073856353759766, -1.1085759401321411, -0.8413130640983582, -1.0715916156768799, 0.31344467401504517, -1.5029960870742798, 0.8328566551208496, -1.1866412162780762, -0.22348928451538086, -0.3710392117500305, -0.523504376411438, -0.6305170655250549, 0.39112621545791626, 1.1299184560775757, 0.3070569336414337, -0.7080978155136108, -0.6624531149864197, -0.012937595136463642, 0.037456411868333817, 1.40934157371521, -0.09391725063323975, 0.5440555214881897, -0.27696964144706726, -1.304402232170105, -1.8331923484802246, 0.28538820147514343, -0.4264203608036041, -1.2030441761016846, -0.32857978343963623, -0.4193838834762573, -1.021019458770752, -1.1116337776184082, 0.4487227499485016, 0.35991522669792175, 0.3505869507789612, 1.2782694101333618, 1.5710121393203735, 1.1551358699798584, -1.0082157850265503, 0.21121083199977875, 2.7502870559692383, 0.2987155020236969, -0.3797354996204376, 0.06134451925754547, 0.3723687529563904, -0.1472419649362564, 1.0182814598083496, 0.5774037837982178, -1.0111737251281738, -0.754489541053772, -0.8041930794715881, -1.323541283607483, -1.262050747871399, 1.2577857971191406, -0.6580538749694824, 1.4643393754959106, 2.536855459213257, 3.0088982582092285, -1.5927717685699463, 0.2179073542356491, 0.14745309948921204, 1.3960193395614624, 0.8199582695960999, -0.6913068294525146, -0.1268925666809082, 0.39250144362449646, 0.6869235634803772, 0.7669679522514343, -0.010874321684241295, 1.7163175344467163, 2.2416608333587646, -1.5829484462738037, 0.13468940556049347, -1.3243204355239868, 0.23859724402427673, 0.13128359615802765, -2.3997116088867188, 1.398283839225769, 0.045623667538166046, -0.334975004196167, 0.29290440678596497, 2.5453155040740967, -1.7064229249954224, -1.459835171699524, -1.2621676921844482, 1.6350293159484863, 1.0793761014938354, -0.9491006731987, -0.602433443069458, 0.8247911334037781, 1.3604005575180054, -0.6068994998931885, -1.2757782936096191, -1.4902807474136353, -1.2502706050872803, 0.3161464333534241, 1.8399738073349, -0.14879190921783447, -0.2998802065849304, -0.5575693249702454, -0.571276068687439, 0.37537050247192383, -0.012687590904533863, -0.5855515599250793, 2.4001688957214355, -0.28597134351730347, -0.7559458613395691, 1.1527363061904907, 0.08570561558008194, -0.9893633723258972, -0.10670679062604904, 0.1863030642271042, 1.1863298416137695, 0.022713633254170418, -1.4853532314300537, 1.6672089099884033, -1.089390754699707, 0.7171657085418701, -0.516578197479248, -0.1750643253326416, 0.661098837852478, 0.6486371159553528, 0.37152889370918274, -0.24195222556591034, 0.8937375545501709, 0.527066171169281, -0.6149929761886597, 0.055375829339027405, 0.10097330063581467, 0.22212213277816772, -0.3459621071815491, -0.41752347350120544, -1.5757975578308105, -1.6993846893310547, -1.568735122680664, 2.2896318435668945, 1.1736907958984375, 1.1569777727127075, -0.15433432161808014, 2.213352918624878, 0.5938555002212524, 0.04635842889547348, -1.325466275215149, 0.31604883074760437, 1.477950930595398, -0.9536774754524231, 0.5303425788879395, -0.14849770069122314, -1.186635136604309, 0.6026149392127991, -0.903040885925293, 0.11870502680540085, 0.9479506611824036, -1.0010747909545898, -0.38422998785972595, 0.2584822177886963, -0.19699546694755554, 0.7922536134719849, 0.6610404849052429, 0.20516572892665863, 0.6392599940299988, 1.7397077083587646, -1.7369639873504639, -0.7035046219825745, -1.061169981956482, 1.1859266757965088, -0.8491966724395752, 0.4226725995540619, -0.10774794220924377, -0.27388638257980347, -0.7711774110794067, -0.018405232578516006, 0.16699866950511932, 1.168067455291748, -0.9007024168968201, -0.5242372751235962, 0.15392272174358368, -0.8391535878181458, 1.790036916732788, -1.1567857265472412, 0.34675130248069763, -1.480170488357544, 0.327960342168808, -0.28876689076423645, -0.40424811840057373, 1.900146484375, -0.6214789748191833, 1.6319690942764282, -0.6989266276359558, 0.7676348090171814, -1.4236011505126953, -0.13667148351669312, 0.6274287104606628, -0.8894578814506531, 0.3978394567966461, -0.7225285172462463, 0.7641862630844116, 1.119572639465332, -1.2723948955535889, 1.4811969995498657, 0.8622677326202393, 0.5866299271583557, 0.6006354689598083, -0.7173332571983337, 0.9701709151268005, 0.5278376340866089, -0.6744493246078491, -0.08012828975915909, 0.2004987597465515, -0.8866293430328369, 6.40950893284753e-05, 0.011193733662366867, 2.0580596923828125, -1.4888639450073242, 0.8839862942695618, -1.3524070978164673, -0.3558211028575897, -2.2451956272125244, -0.1639312356710434, 0.5582655668258667, 0.39245736598968506, -1.757718801498413, -0.8263323307037354, -0.1091446578502655, -0.13531093299388885, -0.6066271662712097, 0.4204540252685547, -0.35010913014411926, 0.4888223707675934, -0.7974464893341064, -0.9291698932647705, 0.8901742696762085, 0.6034936904907227, 0.16957901418209076, 0.7373071908950806, 0.24692392349243164, 0.2863501310348511, 0.12301651388406754, -1.3921470642089844, 0.4843319356441498, -0.21830452978610992, -1.0289543867111206, 0.7965257167816162, -0.3753005862236023, -0.20218680799007416, 0.7585790157318115, -0.2745111882686615, -0.07972735911607742, 0.20051789283752441, -0.3917306661605835, -0.78561931848526, -2.3355705738067627, -0.7320061326026917, -0.6621401309967041, -1.3232643604278564, 1.5855505466461182, 0.29431062936782837, 0.8649402856826782, 0.29955950379371643, -0.7089741826057434, -0.13112588226795197, 0.1223767027258873, -1.53904128074646, -0.32101044058799744, 1.286752462387085, 0.3049221634864807, -0.4420832097530365, -0.7226521372795105, -0.0415433906018734, -0.32546502351760864, -0.3820566236972809, -1.0398885011672974, 1.8838250637054443, -0.004109431058168411, -0.6855719089508057, 2.1055362224578857, -0.30103054642677307, 0.4137531816959381, -0.5160359144210815, 0.04211849346756935, 1.4358136653900146, 0.01836848258972168, -0.253726989030838, -0.05016284063458443, 0.2760070860385895, 1.020296335220337, 0.5767641067504883, -0.9237451553344727, 1.6118414402008057, -0.3484162986278534, 0.6136203408241272, -1.2286179065704346, -1.5118767023086548, -0.49154290556907654, -0.44662001729011536, -0.48976704478263855, 0.372536301612854, 1.4216551780700684, -0.9375239014625549, -0.9925628900527954, 0.18729767203330994, -0.08152635395526886, -0.40153783559799194, 0.07046680152416229, 1.180621862411499, 0.4380349814891815, 0.29787683486938477, -0.17455485463142395, -0.2787454128265381, -1.282849669456482, -1.7012939453125, -0.33550387620925903, -0.49501004815101624, -0.1825505793094635, -0.03431785851716995, -0.7778875231742859, -0.5795835256576538, 2.555528163909912, 0.6731573939323425, -0.5207234621047974, 0.8482123017311096, -0.8288255929946899, 2.0630388259887695, 0.7649244070053101, -0.9437476992607117, -0.15028870105743408, 0.7189111113548279, 1.1779218912124634, -0.6040248870849609, 0.07722141593694687, 1.3937580585479736, -0.3365173935890198, 0.6911414265632629, -0.4183945953845978, 1.0428893566131592, 0.21107232570648193, -1.0932652950286865, -0.009381561540067196, 0.28145891427993774, 1.2916532754898071, 0.1719612032175064, -0.3739338517189026, 0.8102163076400757, -0.24793395400047302, -1.0068753957748413, 0.9170335531234741, 0.6896119713783264, 1.1010832786560059, -1.3936238288879395, -0.6585316061973572, -0.48238927125930786, 0.33851325511932373, -0.3431277573108673, -0.7247498035430908, 0.27190524339675903, -0.6735672354698181, 0.5789512395858765, 0.6316667199134827, 0.14086288213729858, 0.3116040825843811, -2.154712200164795, 0.1252303272485733, -0.12488266825675964, 1.174267053604126, 0.22530345618724823, -1.3456106185913086, 0.2555840313434601, -0.02968914434313774, -0.6768528819084167, 0.9638104438781738, -0.24264322221279144, -1.1700295209884644, -0.569648802280426, 0.6943448781967163, -0.47015202045440674, -2.8800137042999268, -1.4673160314559937, 0.8337334990501404, -0.06923692673444748, 0.08111661672592163, 0.3301530182361603, -1.6531740427017212, 0.9515832662582397, 1.0017297267913818, -0.022151272743940353, -0.5775960087776184, 0.2949684262275696, 0.09073971211910248, 0.195081427693367, 0.38863298296928406, -0.26184943318367004, 0.4718264639377594, 0.29373404383659363, -0.8527772426605225, -0.2807580828666687, 0.98453688621521, -0.9398687481880188, 0.6194832921028137, -0.32416194677352905, 0.8868684768676758, -0.6755548715591431, -0.841655969619751, -0.2887551188468933, 0.8869612812995911, -0.3423583507537842, -0.37739098072052, -0.9079315662384033, -1.2928673028945923, 0.9948837757110596, 0.5636900663375854, -0.2871852219104767, -2.008233070373535, -1.1719900369644165, -0.05962416157126427, 2.256906747817993, 0.02002316154539585, -0.043245285749435425, -0.29188188910484314, 0.3637346029281616, 2.7692127227783203, -0.14557702839374542, -0.06248106434941292, 3.097364664077759, -0.0732521191239357, 0.7074858546257019, -0.7683436274528503, 0.5314653515815735, -0.29297417402267456, 0.10287132859230042, -0.06802839785814285, 0.245303675532341, -0.4922937750816345, 0.978757381439209, 0.4366614520549774, -0.970640242099762, -0.07400260120630264, -0.10917463898658752, -0.3041212856769562, -1.109261393547058, 0.4852314591407776, -0.2596360743045807, -0.7849244475364685, -0.4930304288864136, 0.6112387180328369, 1.451554298400879, 1.3358466625213623, -0.5754258036613464, -0.8252849578857422, 1.0919125080108643, 1.0110046863555908, 0.733421802520752, -0.6895843148231506, 0.9114493131637573, -0.2870621681213379, -0.803776741027832, -0.5616780519485474, 0.4934537410736084, -2.18583345413208, -0.37635356187820435, 1.6315613985061646, 1.6309857368469238, 0.1472858339548111, -0.022878170013427734, -0.28091156482696533, -1.0166618824005127, -2.4516639709472656, 0.5872995257377625, 0.47493284940719604, -0.16699625551700592, -0.9702129364013672, -0.4936845302581787, 0.4983304440975189, -0.5238606929779053, -0.3555656373500824, -0.3536868393421173, 0.6439492106437683, 0.39514103531837463, -0.6542249321937561, -0.11753961443901062, -1.4537625312805176, -0.8774337768554688, 0.8310579061508179, -0.9220204949378967, -0.8317006826400757, -0.9970393180847168, -0.20122340321540833, -0.9844887256622314, -0.3711710572242737, -1.2729535102844238, -0.1599654108285904, 1.8294436931610107, 0.345196932554245, 0.23543107509613037, 1.1595220565795898, 0.5670960545539856, 0.04136068373918533, 1.1631629467010498, 0.920648455619812, -0.10359466820955276, 0.5095133781433105, 0.04141007736325264, 0.9782693982124329, 1.557173728942871, 0.6234772205352783, 1.2815741300582886, 1.777457356452942, 1.4939032793045044, -0.8926141858100891, 0.28280359506607056, 1.0948028564453125, 0.8641477823257446, 0.3782230019569397, 0.3080730140209198, 0.26203060150146484, 1.7012771368026733, 3.485283136367798, 2.469167470932007, 0.9153690934181213, 0.2106994092464447, 1.600948452949524, 0.06683435291051865, -2.5048351287841797, 2.0648574829101562, -1.1665396690368652, -1.699065089225769, -0.8412451148033142, 1.9608792066574097, -0.03872675076127052, -0.51082843542099, 0.12490823864936829, -1.4232221841812134, -1.2790818214416504, -1.8024795055389404, -0.602950394153595, -1.1611968278884888, -0.9011206030845642, 0.635383665561676, -1.1841578483581543, -0.01165224239230156, 0.08371153473854065, 0.28097787499427795, -0.46533894538879395, -1.1530057191848755, -1.5272767543792725, -0.619030237197876, -0.47270089387893677, -1.1106022596359253, 0.6838074922561646, -0.8279728889465332, -1.189937710762024, 0.3550170361995697, 0.7992644309997559, -0.9053036570549011, 0.820389986038208, -1.23714017868042, -2.282883405685425, -1.0858001708984375, -1.5245251655578613, -0.09568886458873749, -0.2711297571659088, -0.7236716151237488, -0.5447278618812561, 0.2690529227256775, 0.9989210963249207, -0.08974793553352356, -1.2750203609466553, -0.10716617107391357, -0.5957357287406921, 0.27301815152168274, 0.009025760926306248, 1.1971263885498047, -1.6288783550262451, -1.2634377479553223, -0.3675670325756073, -1.6238560676574707, -0.8171499967575073, 0.3861124813556671, -1.6313599348068237, -0.5299047827720642, 0.5632660388946533, 1.6653164625167847, 1.530841588973999, -0.5682277679443359, 0.04062364250421524, 1.325113296508789, -1.037688970565796, -0.32843470573425293, -0.8205786347389221, -0.6239820718765259, 2.713524580001831, -0.7012240886688232, 0.5877070426940918, -0.5739914774894714, -0.2668692171573639, -1.5968290567398071, 0.7372618317604065, -0.1341741681098938, -0.2921634018421173, 1.005491018295288, 0.19112160801887512, -1.5823253393173218, -3.434999704360962, 0.4693104028701782, -0.5215943455696106, 1.468285322189331, 2.4192793369293213, 0.30846819281578064, 0.7286432385444641, -1.7566221952438354, 0.8554051518440247, -1.1036499738693237, -0.25210317969322205, -0.7681014537811279, -1.100185513496399, 0.11423499882221222, 0.3505367636680603, -0.5424026846885681, 0.13726037740707397, -0.810991108417511, 2.5169031620025635, 0.3118710517883301, 1.1284376382827759, -0.13595475256443024, 1.1324290037155151, -1.0415778160095215, -1.093079686164856, -0.2338019162416458, 0.09107045084238052, -0.5421499013900757, 1.265785574913025, -0.94020676612854, 1.2198468446731567, -1.0864571332931519, 0.9018782377243042, -0.0922742560505867, 0.44224998354911804, 2.0939791202545166, 1.8053667545318604, -2.0523977279663086, 0.030140824615955353, -1.4209933280944824, 0.45071354508399963, -0.5427722930908203, -0.15061067044734955, -1.219009280204773, 0.5621603727340698, -1.1959521770477295, 0.8196296691894531, 0.2066052258014679, -1.6532570123672485, 1.5173722505569458, -1.6401622295379639, 2.184506416320801, -0.17709636688232422, 0.26263728737831116, 0.5390433073043823, -0.7668975591659546, 0.9450200200080872, -0.0894932746887207, -2.475827693939209, 0.2603019177913666, 0.7734342217445374, -0.360156774520874, -0.6855209469795227, -0.1888805776834488, -0.8567911386489868, 0.8909458518028259, 1.1998794078826904, 0.012341535650193691, -0.253858357667923, -0.9711118340492249, -0.18016865849494934, -0.09581678360700607, -0.42801690101623535, -1.1733051538467407, -1.1204438209533691, 0.942687451839447, 0.6222891211509705, 1.575140357017517, 1.0431652069091797, 0.7050366401672363, -0.23142120242118835, 0.6076921820640564, -0.560301661491394, 0.2702941298484802, -0.14345894753932953, -1.670609474182129, 0.15890632569789886, -1.5378772020339966, -0.12632037699222565, 0.7460969686508179, 0.443805992603302, -0.8008652925491333, -0.43258869647979736, -1.108564853668213, 1.3746343851089478, -0.5786333084106445, 1.6646236181259155, 0.8868343830108643, -0.6169006824493408, 0.09840252250432968, -0.2538590133190155, -0.40575262904167175, -0.7110366821289062, 0.6305794715881348, -1.0540074110031128, -0.7608382105827332, 0.6395723819732666, -0.5322109460830688, 0.16798612475395203, 1.1187162399291992, -0.030393412336707115, 0.6264179348945618, -1.9110403060913086, 0.24705460667610168, 1.1240980625152588, 0.14455938339233398, -0.9381725788116455, 0.5909331440925598, -0.5351579785346985, 0.6156412959098816, 0.8454234600067139, 1.119293451309204, -1.3407357931137085, -0.6911052465438843, -1.074958086013794, -1.3657792806625366, 1.3402576446533203, 0.6298496723175049, -0.29876139760017395, 1.0872654914855957, 1.1879472732543945, -0.4845355451107025, -0.639085054397583, -0.08451259881258011, -0.9919989705085754, -0.6704412698745728, 0.8215338587760925, -1.710956335067749, -0.31777456402778625, 1.9402133226394653, 0.0677262395620346, 1.2649205923080444, 0.4640261232852936, 0.6667823791503906, -0.8706793189048767, 0.055400777608156204, -1.144241213798523, 1.2936809062957764, -0.6912825703620911, 1.4164819717407227, -0.13485725224018097, -1.1361018419265747, -0.47357437014579773, 0.6648595929145813, 1.607153296470642, 0.1242438331246376, 0.528071939945221, -1.0925743579864502, 2.0454890727996826, 0.5876902341842651, 1.116822361946106, -0.5161398649215698, 0.8901268839836121, 0.1996525079011917, 0.5455771088600159, -0.6753847002983093, 0.3950191140174866, 0.49663785099983215, 0.4833414554595947, -0.3509812355041504, 1.0444849729537964, -0.0374571792781353, -0.6622315049171448, 0.02532222494482994, -0.23531897366046906, 1.2978512048721313, 0.49779751896858215, -0.3000282943248749, 0.2972222864627838, 0.2760045528411865, -1.411129117012024, -1.113832950592041, -0.7590942978858948, -0.33194202184677124, -0.26739227771759033, -0.15525436401367188, -0.06821285933256149, 1.1424682140350342, 1.3924107551574707, -1.676076889038086, 1.9798020124435425, 0.5892133116722107, -1.6097708940505981, -0.04521983861923218, 0.2973572909832001, 0.16529671847820282, -0.3099251389503479, -0.8126372694969177, 0.5697189569473267, 0.696366012096405, 0.9356598854064941, -1.3874890804290771, -0.567625880241394, 0.7620988488197327, -0.09884793311357498, 0.8478305339813232, 1.2658584117889404, -0.2883080542087555, 0.056131672114133835, -0.10085397213697433, -0.9913709163665771, 0.6856256723403931, 0.45804691314697266, 0.42756614089012146, 1.486925482749939, -0.5293633937835693, 0.1529739946126938, -0.4927327632904053, -2.6620888710021973, 0.7213801741600037, -1.7107805013656616, -0.49529367685317993, 0.8731954097747803, -0.6127130389213562, -0.5140348672866821, 0.7189128398895264, -1.2791500091552734, 0.13569577038288116, 0.026817621663212776, 2.476759195327759, -0.13737085461616516, 0.416908860206604, -0.7187778949737549, -0.318724125623703, 0.20565427839756012, -1.820290207862854, 1.3415567874908447, 0.035635046660900116, -0.26006999611854553, -1.1998943090438843, 0.2657584249973297, -0.4019109904766083, -0.044123850762844086, 1.1023850440979004, 0.39318767189979553, 0.23400035500526428, 0.5758318305015564, -0.27193596959114075, 1.143135666847229, 0.5847746729850769, 0.2295866757631302, -1.2148791551589966, 0.8489974141120911, 0.3037557899951935, 2.1501457691192627, 0.24597039818763733, 0.6416022777557373, -0.9047678112983704, -0.5238809585571289, 0.9497052431106567, -0.28218790888786316, 2.0000407695770264, -1.6332403421401978, 1.1636345386505127, -0.012569497339427471, -1.0204482078552246, -2.2251644134521484, -1.9266061782836914, 0.6268717050552368, -1.7592004537582397, -1.211380958557129, -0.8755974769592285, -0.5759416222572327, 1.0756934881210327, 0.8457658886909485, -0.6166600584983826, -0.9642490148544312, -0.20583568513393402, -0.26325875520706177, -0.8966211080551147, -0.2916775345802307, 0.6393640041351318, -0.6400126218795776, -0.1789432317018509, -0.6403469443321228, -1.043198823928833, 1.2659380435943604, 0.46792492270469666, 1.0971214771270752, -0.5432916879653931, 1.2128493785858154, -1.4063102006912231, 1.593699336051941, 2.1887080669403076, 0.0013767627533525229, -0.09556275606155396, 0.3150688409805298, -0.07761082053184509, 1.036323070526123, 1.3092553615570068, -0.8113617300987244, 0.5802773833274841, -1.343043327331543, -0.8027242422103882, 1.5310300588607788, -2.6104018688201904, -0.5161641836166382, -0.024922726675868034, 0.03330550342798233, 1.0742251873016357, 0.4974551796913147, 0.6101702451705933, -0.9556429386138916, 0.34643250703811646, -0.1483778953552246, -0.9552869200706482, 0.01880207657814026, 0.17147299647331238, -0.24114234745502472, -0.09548655152320862, 1.1521438360214233, -0.38651397824287415, -0.5354372262954712, 1.9160867929458618, 1.2930878400802612, -0.3332473635673523, -0.9036173224449158, 0.52490234375, 0.6313815116882324, -0.09973830729722977, -0.10548392683267593, 0.5067489743232727, 0.24618132412433624, -1.0138649940490723, -0.5607947111129761, -0.255331426858902, -0.3711502254009247, 0.47821828722953796, -0.0967341884970665, -0.608951985836029, -0.709206759929657, -0.14028917253017426, -0.04491022229194641, -1.1654701232910156, 0.5196730494499207, -1.844399094581604, 0.028213422745466232, 1.5581421852111816, -0.13161559402942657, -0.17673148214817047, 0.8982906341552734, 0.8320261240005493, -0.12833446264266968, -0.07157298922538757, -0.7255738973617554, -1.0052640438079834, 0.012735051102936268, 0.1026616171002388, -0.3810415267944336, 1.228585958480835, -1.4484210014343262, -0.20075950026512146, 0.3964025676250458, -0.6259799003601074, -1.022914171218872, -0.2828795611858368, -1.2023571729660034, 0.8589178919792175, 0.6855300068855286, 0.8046875, 2.377185344696045, -0.03792779892683029, -2.713343858718872], [-0.9095465540885925, -0.37237000465393066, -0.8028228282928467, 0.06452963501214981, -1.0883179903030396, -2.1205945014953613, 0.909952700138092, 0.919999361038208, -0.6651995182037354, -1.5149402618408203, 0.2705997824668884, 0.5472949743270874, 0.5876458883285522, 0.2656959295272827, -0.9722644090652466, 1.049382209777832, 0.9519924521446228, 1.19044029712677, -0.9554470777511597, -0.705176055431366, -1.335323691368103, -1.7124954462051392, -0.6129856109619141, 0.612237274646759, 0.1299663931131363, 0.15637390315532684, 0.12200450152158737, -1.6613062620162964, 2.09268856048584, -0.3932895064353943, -0.12519998848438263, 0.6584154367446899, -0.7508013248443604, 1.67038094997406, 1.7401368618011475, -0.07702109217643738, -0.7258846163749695, 1.4049654006958008, -2.122701406478882, -1.2711313962936401, 0.06159418448805809, -1.2884474992752075, 1.0577651262283325, -0.0038476544432342052, 0.13309916853904724, 2.2571499347686768, 1.6804567575454712, -0.16308921575546265, -0.13015292584896088, 0.0021904315799474716, 0.1399581879377365, -0.09975069016218185, 0.12914448976516724, -0.09777458012104034, -1.020706295967102, -0.8519850969314575, 0.5158821940422058, -0.9932809472084045, 0.08911275118589401, -0.1870967596769333, 0.7535417675971985, 0.7051098942756653, -2.5475451946258545, 0.04104580730199814, 0.6928893327713013, 0.2695508599281311, 0.010259317234158516, -0.06465515494346619, -2.358715295791626, 0.3022385537624359, 0.16961103677749634, 0.2658030688762665, -0.4220087230205536, -0.001181798754259944, -2.1378660202026367, 0.9415423274040222, 1.0976296663284302, 0.25798723101615906, 1.936134934425354, 1.257725715637207, -0.18999417126178741, 0.06911289691925049, -0.7149535417556763, 0.8379297852516174, -0.9566331505775452, -1.4015828371047974, -0.5856546759605408, -0.48278748989105225, -0.5828074812889099, -0.4905599355697632, 0.4633603096008301, -0.8385407328605652, 1.5127102136611938, 0.5854876041412354, -0.14301708340644836, -1.8148871660232544, -0.37316930294036865, 0.7077546715736389, -0.34040480852127075, -0.19220668077468872, 0.5934382081031799, 1.0860105752944946, -1.6115388870239258, -0.0031193674076348543, -0.16201603412628174, -1.9334548711776733, -0.30108001828193665, 1.105796217918396, -0.7089260816574097, -0.287925660610199, -1.594303011894226, 0.6723558306694031, 2.492170572280884, -0.13070723414421082, 0.3413732647895813, -1.0204359292984009, -0.03235277161002159, 1.5589475631713867, 0.12480680644512177, -0.6114323735237122, 1.3444722890853882, 0.8050835132598877, -1.6895451545715332, -0.34149426221847534, 0.624735414981842, -3.1854612827301025, 1.3607834577560425, 1.0207390785217285, -1.3967798948287964, 0.5996575355529785, 1.620391845703125, 1.9768773317337036, -0.5280536413192749, -1.7551801204681396, 0.6879245638847351, 0.7729555368423462, -1.055783987045288, -0.6912636160850525, -0.7931947112083435, -0.06832388788461685, -1.5705938339233398, -1.636438012123108, 1.5869439840316772, -1.3796759843826294, -1.200835108757019, 1.0516116619110107, -0.260271817445755, -0.2566092312335968, 0.5743792653083801, -0.9180759787559509, -0.6897647976875305, 0.1442934274673462, -1.1291862726211548, 0.4226675033569336, 0.7695968747138977, -0.6550657749176025, -0.14187458157539368, -1.0183228254318237, 1.1030393838882446, -1.1479908227920532, 0.8187914490699768, 1.0939292907714844, 1.7419811487197876, -0.6753486394882202, -1.5439198017120361, -1.2349976301193237, -2.367518901824951, -0.04212508350610733, -0.06001356244087219, 0.2932587265968323, -0.165348619222641, 0.1827264279127121, 0.47761911153793335, 2.870955228805542, 0.4090653955936432, -1.8324047327041626, -0.0719943642616272, 0.31375083327293396, 2.2056872844696045, 0.5259852409362793, 0.572303831577301, -0.8849592208862305, -0.2653462886810303, 0.8745099902153015, 0.87394118309021, 0.37086760997772217, 0.7590839266777039, -1.2876991033554077, -0.5921221375465393, 0.1928267478942871, -0.3662368059158325, 0.43424904346466064, 0.6357677578926086, 0.30023932456970215, -0.8710494041442871, -0.2882751524448395, -2.4324827194213867, 0.956805408000946, 0.0060822805389761925, 0.17120468616485596, 0.12402941286563873, -1.2265983819961548, 0.9061896204948425, -0.2749606966972351, 0.6376987099647522, -0.3068088889122009, 0.05152535066008568, -0.7120770215988159, 1.509606122970581, 1.3657629489898682, 0.6092114448547363, -1.8886890411376953, -0.1625441163778305, 0.9987632632255554, 1.0051277875900269, 0.10345610976219177, -0.4887273907661438, -1.1920686960220337, -0.4350358843803406, -1.3919271230697632, -0.34970417618751526, 0.7378517985343933, 0.5569467544555664, -0.8800660371780396, -0.2654320001602173, 1.1037272214889526, -0.026038046926259995, 0.7202375531196594, -0.2936360836029053, -0.22135213017463684, -0.45499947667121887, 0.5575221180915833, 0.523312509059906, 0.4079791009426117, -0.147390678524971, -1.2196953296661377, -1.9056813716888428, -1.5563377141952515, -0.46859630942344666, 0.10384554415941238, 1.1475416421890259, -1.1008784770965576, -0.5910199284553528, 0.11580498516559601, -0.5397791862487793, 1.674331545829773, -1.524023175239563, -0.7413364052772522, -0.2798847556114197, -0.027062444016337395, 0.4770951271057129, 1.4487271308898926, -0.5047857761383057, -1.3284906148910522, -0.3841375410556793, 1.7806857824325562, 0.08212093263864517, -0.9857887625694275, 0.9080888032913208, -0.3896055519580841, 1.1943992376327515, -1.2504751682281494, 0.10817008465528488, -0.7255645394325256, -1.5407525300979614, -0.6626775860786438, -2.993020534515381, -0.6280204057693481, 0.3316299617290497, -1.1680500507354736, -1.526626706123352, -0.4942842125892639, 0.13000018894672394, 0.5431311726570129, -1.2329021692276, 0.04012013599276543, 0.6430912017822266, 0.7438243627548218, -0.8939996361732483, 0.6841418743133545, 1.6819454431533813, -0.21596860885620117, 0.19317036867141724, -0.9142070412635803, -0.06001678481698036, -0.885361909866333, -0.9997252225875854, 2.481924295425415, -2.0908946990966797, 0.5643941760063171, -0.03607289493083954, 0.43907395005226135, 0.09312385320663452, 0.09458432346582413, 1.67967689037323, 0.10264869034290314, -0.09825880080461502, 0.9273634552955627, -0.3156581521034241, -0.23706552386283875, 1.6638717651367188, -1.3354703187942505, -1.2660250663757324, -1.346397876739502, -1.8140275478363037, 0.9915226697921753, -1.48542058467865, -0.21226860582828522, -1.3286595344543457, -1.464158535003662, -0.3465108871459961, 0.1387476772069931, 0.02759828045964241, 0.846271812915802, -0.06835257261991501, 1.137174129486084, -0.8434524536132812, 1.3748618364334106, 0.0412101112306118, -0.6967353820800781, 1.074126958847046, -0.04905875027179718, -0.8144770860671997, 1.4891579151153564, -0.7930825352668762, -1.3555604219436646, -0.22705544531345367, -0.27712708711624146, 0.1316741555929184, -1.1596016883850098, -0.21435409784317017, 1.926448941230774, -1.6047295331954956, -1.5903476476669312, 0.4992954730987549, -0.5902134776115417, 0.8646222352981567, 0.2388269305229187, -1.812469244003296, 1.830372929573059, 1.1615278720855713, -0.4319343566894531, -0.26689624786376953, -1.0635192394256592, 0.31623390316963196, -1.3929643630981445, 0.8601001501083374, -0.7327256202697754, -0.45560991764068604, -1.1916099786758423, 0.171725332736969, -0.7349697351455688, 1.053810477256775, -0.850507915019989, 0.552908182144165, -0.798438549041748, -0.11262709647417068, -1.758065104484558, -1.042629361152649, -0.8054121136665344, -0.7828440070152283, -0.4482530653476715, 1.8501516580581665, 1.8896652460098267, 0.6670924425125122, 0.034900568425655365, 0.10340071469545364, 1.4605075120925903, 0.10951505601406097, -0.9735191464424133, -1.1336027383804321, -1.6545566320419312, 0.04554108530282974, 0.08195272833108902, 0.8752102851867676, 0.5393837094306946, 0.3857021629810333, 0.214169442653656, -1.5686514377593994, -1.046905755996704, 0.4152548313140869, -1.4974935054779053, 0.11523488909006119, 0.7950263023376465, -2.15754771232605, -0.9159108400344849, 0.8669638633728027, 0.38302233815193176, -0.644364058971405, 2.7851502895355225, -0.26382118463516235, -0.019118282943964005, -1.78958261013031, -0.48224204778671265, 0.555721640586853, 0.7788029313087463, 0.8702021837234497, 0.06303390860557556, -2.0566272735595703, 1.2969064712524414, 1.0670634508132935, 0.5749580264091492, 2.3177073001861572, 1.1808933019638062, 0.930332601070404, 0.8682172298431396, 0.2131614089012146, -0.3709130883216858, -0.5067687034606934, -1.4084442853927612, 1.2298836708068848, 0.7344292998313904, 0.38811540603637695, 0.6129410266876221, -0.46398234367370605, -2.2259433269500732, -1.1215476989746094, 0.13433274626731873, 0.412977397441864, -0.5942787528038025, 1.5246820449829102, -0.12988518178462982, -0.9620888829231262, 1.154937505722046, 1.816604733467102, -0.9783030152320862, -1.3018872737884521, 0.09940919280052185, -0.7591845393180847, -0.1029483899474144, 0.637921929359436, -0.9396858811378479, -1.0973423719406128, 0.6643138527870178, 1.8801140785217285, -0.497732937335968, -0.031404610723257065, 0.6164300441741943, -0.2883891761302948, 0.2541523277759552, -0.8166528940200806, 0.4945848286151886, -0.5528554916381836, 1.5127917528152466, -0.33182621002197266, -0.7487252950668335, 0.09233531355857849, -0.5461041927337646, -0.06087585166096687, 0.8934495449066162, 0.6267057061195374, 0.7872872352600098, 0.36427927017211914, 0.6460316777229309, -0.06046782061457634, -0.5710182189941406, 0.6221267580986023, 0.8677104711532593, -0.25055697560310364, -0.07250847667455673, 0.06380166113376617, 0.6564157009124756, 0.7677951455116272, 0.5890946984291077, -0.9672918915748596, 1.7127071619033813, -0.2457626312971115, -0.08768370747566223, 1.2743539810180664, 0.21384353935718536, -0.08971796929836273, 1.2843613624572754, -0.09526533633470535, 1.3062384128570557, -0.22703810036182404, 2.0027949810028076, -1.4539484977722168, -0.4660319685935974, 0.8403409719467163, 0.7284662127494812, 1.2624437808990479, 0.5962799191474915, -0.4175107777118683, -1.1700702905654907, 0.40918174386024475, 0.3733232319355011, 1.2417839765548706, -0.7051004767417908, -0.5327149033546448, 1.2306632995605469, 1.5355170965194702, -2.023489475250244, -0.6099767684936523, -1.3414897918701172, -0.7888249158859253, -1.0231842994689941, 0.2782635986804962, -0.16822201013565063, -1.6408568620681763, -0.2806580066680908, -1.1451606750488281, -1.5373833179473877, -0.929715633392334, -3.036374092102051, -0.8096851706504822, -0.0413687638938427, 1.6678524017333984, 0.1284283548593521, -0.3904649019241333, -2.6161532402038574, 1.6755794286727905, -0.8652600646018982, -0.19595922529697418, 0.1614076942205429, -0.33367815613746643, -0.5502949953079224, -0.5002081990242004, -0.24141596257686615, 1.223124384880066, -0.6018733978271484, -1.504844069480896, 0.3670535981655121, 1.3139079809188843, 0.17826960980892181, -0.32797056436538696, 0.8507109880447388, 0.9520215392112732, -1.5679270029067993, -0.64156174659729, 0.4405880868434906, -0.08548922091722488, 0.2086549699306488, 0.7784025073051453, 0.08152797073125839, -0.7542299032211304, 1.8210866451263428, -0.8400934338569641, 1.187247395515442, -0.9794793725013733, 0.9039071202278137, -0.03859658166766167, -0.44441038370132446, 0.8065073490142822, 0.07776185125112534, -0.33165243268013, 0.9955843091011047, -1.2928344011306763, 2.003772020339966, 0.007779079023748636, 1.1108746528625488, 0.449415922164917, 2.0206663608551025, -0.3225906789302826, 0.33786916732788086, 1.8093348741531372, -0.5448293685913086, -0.30360251665115356, 1.130954384803772, 1.1539762020111084, -0.6292355060577393, 0.6190466284751892, 0.9430825114250183, 0.3563883900642395, -1.1449061632156372, 0.1739228218793869, -0.3287569582462311, -0.36300358176231384, 1.163321614265442, 0.07428917288780212, 0.8358856439590454, 0.688736081123352, 1.3270344734191895, -2.0210041999816895, 0.12990084290504456, -0.6248855590820312, -1.7619471549987793, 0.19458989799022675, -0.2877410352230072, -0.7576330900192261, 0.6451925039291382, -1.432673692703247, 0.11811770498752594, 0.5181940197944641, 0.5086104869842529, -0.6309149861335754, 1.8132236003875732, 0.01801658235490322, 0.8498353362083435, 1.818603515625, 1.0655266046524048, -0.5736749768257141, -0.9784465432167053, 1.0686160326004028, 0.7875110507011414, -0.2346092015504837, -1.440822958946228, 0.4431086480617523, 0.10338010638952255, -0.6037643551826477, 0.8207040429115295, -1.0509368181228638, 0.017511524260044098, 0.6509621739387512, 0.6789092421531677, 0.41549256443977356, -2.1939897537231445, -0.9426411986351013, 0.21811814606189728, -0.17881016433238983, -0.6132190227508545, -0.9875908493995667, -1.871315598487854, 0.0224434994161129, 1.3626816272735596, 0.031756557524204254, 0.4024735987186432, 1.063914179801941, 0.1418326050043106, 1.1195449829101562, 0.1071363165974617, 0.43051013350486755, 1.9143158197402954, -0.009744117967784405, -0.8860684633255005, 0.84748375415802, 1.2544479370117188, 0.2882823646068573, 1.6823432445526123, 0.31432193517684937, 0.20198622345924377, -1.191125750541687, -0.3826517164707184, -1.1041606664657593, 1.2060117721557617, 0.155290424823761, -0.09696391969919205, -0.0627792701125145, -0.16323916614055634, -0.19419102370738983, -0.8164246678352356, 1.7605646848678589, 0.809066116809845, 0.48705071210861206, -1.117667555809021, 0.6293979287147522, -0.9455462098121643, 2.4390265941619873, -1.2399494647979736, 0.37469619512557983, -0.8718480467796326, -0.7496396899223328, -0.3825094699859619, -0.04347289353609085, -1.2981008291244507, -0.10870213061571121, -0.6338533163070679, 0.20771922171115875, -1.3908556699752808, 0.5101332068443298, 1.9134451150894165, 0.0747549906373024, 0.6085882782936096, -0.8937658071517944, 0.13292506337165833, -0.1304665356874466, -0.137324258685112, -0.5127657055854797, 0.6590706706047058, -0.5390920639038086, -1.9910658597946167, 2.8863260746002197, 0.5735226273536682, 0.30453962087631226, -0.4170646071434021, 1.4021692276000977, -0.8890910148620605, -1.0725150108337402, -1.461147665977478, 1.0879323482513428, -0.5535039305686951, -1.283583402633667, 0.4786744713783264, -1.711074948310852, -1.0942280292510986, -1.1252926588058472, 0.47489631175994873, -0.0075463359244167805, 0.0006016390980221331, 0.48407411575317383, -1.0659122467041016, 0.17355738580226898, 1.1176849603652954, 1.5266262292861938, -1.092677354812622, -0.031932320445775986, -0.33667469024658203, 0.4081748127937317, -0.0604279600083828, 1.0728973150253296, 0.22831086814403534, 0.25312545895576477, 0.09045993536710739, -0.13503940403461456, 0.7997922301292419, -0.07140468806028366, 0.3662152886390686, 0.6873294115066528, 0.9841099977493286, -0.35620880126953125, 3.058393716812134, 0.6484984755516052, 0.6472064852714539, -2.2935268878936768, -1.338189721107483, -0.4907088279724121, 1.605851650238037, 1.5073115825653076, 0.992939293384552, -1.5430108308792114, -0.42320746183395386, -0.8578093647956848, -0.708922803401947, 0.5097880363464355, -0.1065523773431778, -0.37607651948928833, -2.6826281547546387, -0.17791536450386047, 0.674441933631897, 0.04596704617142677, 0.38064414262771606, -0.3156678080558777, 1.4706872701644897, -0.24378474056720734, 1.5220006704330444, -0.33920061588287354, 0.7069185376167297, -0.17269231379032135, 1.8652689456939697, 1.0520421266555786, -0.9156488180160522, -0.5429656505584717, 0.9990281462669373, 0.23425936698913574, 1.6521550416946411, -1.224525809288025, -0.0856608897447586, 0.5036270022392273, 0.5097697377204895, 1.7119569778442383, -0.6128920316696167, 0.39016079902648926, 0.8682937622070312, -0.9418526887893677, 0.5653712153434753, -1.1890724897384644, -0.9653946161270142, -0.853349506855011, -0.8426517844200134, -0.3434887230396271, 0.43758875131607056, -1.1749902963638306, -0.06006127595901489, 0.2746887505054474, -0.45936450362205505, 0.7160127758979797, 1.1357829570770264, 1.0765451192855835, -0.3878647983074188, 0.1000504195690155, 0.7100765109062195, -0.1858874261379242, 0.339472234249115, 0.4272215962409973, 0.34681057929992676, 0.17252667248249054, -0.21313202381134033, 0.013956625945866108, -0.4970834255218506, 0.11129835993051529, -1.2140023708343506, -0.05503513664007187, 0.4541614353656769, 1.3197968006134033, 0.6650758385658264, 0.9924055933952332, -0.808322012424469, 1.3295159339904785, 0.5081806182861328, -1.2104051113128662, 1.220574140548706, -0.7559853196144104, -1.4399194717407227, -0.2111264318227768, -1.153239369392395, -1.0372430086135864, -1.9003574848175049, 2.4948744773864746, 0.9480072259902954, -1.9563332796096802, 1.3032821416854858, 0.6262388229370117, 0.5464023351669312, -0.9784431457519531, -0.9735404253005981, 0.29849937558174133, 0.1465565711259842, -1.5418360233306885, -0.17681999504566193, -0.8041731715202332, 1.0324337482452393, -0.8621701598167419, -0.15060864388942719, 0.41884341835975647, -0.7052791714668274, -0.6137050986289978, 0.8876709938049316, -1.101716160774231, 0.28376418352127075, 1.0790945291519165, -1.302153468132019, 0.6460719704627991, 0.48275086283683777, 0.28831344842910767, -0.5170478820800781, 1.3590199947357178, 1.5082885026931763, -0.5103179812431335, -0.24706867337226868, 0.6104665994644165, -0.8391514420509338, -0.0850038155913353, 0.7153098583221436, 0.5783806443214417, 0.41335904598236084, -0.9510253667831421, 0.06152122840285301, 0.8453248739242554, 1.111410140991211, -2.3727927207946777, -1.378112554550171, -0.1506345123052597, -1.1874237060546875, -0.6669784188270569, 0.29855743050575256, 0.9053900241851807, 0.9698314666748047, 0.9142733216285706, -0.3374027907848358, 1.244474172592163, -1.4394806623458862, -0.8387339115142822, 1.7557655572891235, -0.026649879291653633, -1.3868650197982788, -0.8765272498130798, -0.6453538537025452, 1.5787098407745361, -0.02462298423051834, 0.25765877962112427, 0.316228449344635, 0.4610408842563629, -1.361182451248169, 0.46875667572021484, -0.9901608228683472, -0.23683932423591614, 0.18839827179908752, -0.9894332885742188, 1.003786325454712, -0.07792188227176666, -1.8653923273086548, 1.3739169836044312, 1.8030303716659546, 2.484999656677246, 0.26921504735946655, -0.3836304545402527, -0.059950947761535645, -1.2073944807052612, -0.09122750908136368, -0.2491409182548523, 0.5055835247039795, 0.35919612646102905, -1.6541719436645508, 0.8536946773529053, 0.6153450608253479, -0.5100862979888916, -0.12829448282718658, 1.1389613151550293, -1.6631925106048584, 0.5377489328384399, -0.33653023838996887, 0.29210522770881653, 0.4949263632297516, -1.5991097688674927, -0.16594527661800385, -0.1405554711818695, -1.3419774770736694, 1.3863438367843628, -0.06883477419614792, 1.89849853515625, 1.7737892866134644, -0.8215799927711487, 1.5534310340881348, 0.844419002532959, -1.1660469770431519, 0.4874107837677002, -0.05726504698395729, 0.7680373787879944, -0.727066695690155, -0.5444755554199219, 0.06320767849683762, 0.697437047958374, -0.1628633737564087, 1.389088749885559, -1.2747737169265747, 1.3700846433639526, -0.0826871246099472, 0.9952811002731323, 0.38126999139785767, 0.9858608245849609, 1.1846541166305542, -0.8329041004180908, -0.055624764412641525, 0.9490732550621033, -1.123396396636963, -0.2579343914985657, -0.3919095993041992, 0.4470472037792206, 2.284165620803833, 1.1512784957885742, 0.25400838255882263, 0.5567260384559631, -0.22414085268974304, 1.688239336013794, 0.3959757089614868, -0.5037503242492676, -0.7909921407699585, 2.2669286727905273, -1.0712852478027344, -1.5206573009490967, 0.2530137896537781, -0.5574513077735901, -0.8283993005752563, -0.1511836051940918, 0.20969483256340027, 1.60258150100708, -0.1871868371963501, -0.82744961977005, -1.5555340051651, 0.3984173536300659, -0.22875230014324188, -1.5718724727630615, 0.1599438488483429, -1.570326566696167, -0.6594008207321167, -0.8048553466796875, 0.21023531258106232, 2.2500789165496826, -1.491564393043518, 1.1793278455734253, 2.073368787765503, -1.137325406074524, 0.31770309805870056, -0.7762925028800964, 0.9386493563652039, -0.12536554038524628, 1.4455273151397705, -0.2545390725135803, 0.029617102816700935, 1.6267293691635132, -1.0976271629333496, -0.5585739016532898, -1.056057333946228, -2.0093131065368652, -0.9258756637573242, 0.24988673627376556, -0.002287001349031925, 0.17907944321632385, 1.4680531024932861, 0.4959326386451721, 0.2977915406227112, -0.10337305814027786, -1.570613145828247, 1.0448429584503174, 0.4294559061527252, 2.0652520656585693, 1.1292436122894287, 1.6122947931289673, 0.5684813857078552, 1.2883524894714355, -1.3573871850967407, -0.5197879076004028, -0.836974024772644, -0.8804358839988708, 2.990078926086426, 0.1414410024881363, -0.055583175271749496, 1.540380835533142, 1.6024352312088013, 1.2681126594543457, -0.958486020565033, 0.07330936193466187, -1.155896544456482, 0.5118986368179321, -0.991179347038269, -0.8587631583213806, -1.2507216930389404, 0.8204950094223022, -1.2387834787368774, -0.8649691939353943, -0.3026482164859772, 0.5638364553451538, 0.9311841130256653, 0.982928454875946, -1.0449343919754028, 0.2779260575771332, 0.22056350111961365, -0.03566106781363487, 0.6711581945419312, -1.1385072469711304, 1.2209084033966064, 0.03162224218249321, -0.2210017889738083, 0.6444692611694336, 0.485693097114563], [-0.3999112546443939, -0.8217321038246155, 0.32543930411338806, -0.007270685862749815, 2.031557321548462, 0.1873505413532257, 0.9906835556030273, -0.8736706376075745, 1.1787453889846802, 0.8388783931732178, 0.20020301640033722, -1.6112747192382812, 0.5908578038215637, 1.0495778322219849, 0.09483268857002258, 0.12082993984222412, -0.05391507223248482, 1.4738876819610596, -0.42176100611686707, 0.9936650395393372, -0.5783560872077942, -0.28057390451431274, -0.6500717401504517, 0.8935060501098633, 0.530842661857605, 0.5642337799072266, -1.27198326587677, 0.9707265496253967, -0.8935350775718689, 0.06448785960674286, -0.6480483412742615, 0.4651769697666168, -0.0932517722249031, 1.0167021751403809, -0.3325102925300598, 1.4597772359848022, -0.35902687907218933, -1.2489100694656372, -1.1446269750595093, -0.40139299631118774, 1.0824450254440308, 0.1319165676832199, -1.798909306526184, 2.444944143295288, 1.5324811935424805, 0.48770463466644287, -0.7050479054450989, -0.19362959265708923, -0.004840999376028776, 1.6238728761672974, 0.020483197644352913, -0.8032536506652832, 0.12413661181926727, -0.4294467270374298, 1.267241358757019, 0.18380895256996155, -0.5908270478248596, -1.988556981086731, -0.824867844581604, -0.21389736235141754, 1.5866785049438477, 0.18191495537757874, 0.02660967968404293, 1.428497076034546, -0.3939509689807892, 0.3459760546684265, 2.1804251670837402, 0.33889031410217285, -0.23857000470161438, 3.1129660606384277, -0.288289338350296, 0.4535151422023773, 1.609217643737793, -0.6179532408714294, -0.6030442714691162, 0.006203680299222469, 0.021416153758764267, 2.6966512203216553, 1.0636532306671143, -1.0724042654037476, -0.06234074756503105, 1.3513530492782593, -0.6119190454483032, -1.2623217105865479, -0.16876932978630066, 0.24385495483875275, 0.5875347852706909, 0.9689643979072571, 0.23873598873615265, 1.038968563079834, -1.3008781671524048, 1.2499397993087769, 0.042345162481069565, -0.4180355668067932, -0.8345754742622375, -0.05930783227086067, -0.925722599029541, -0.8232592940330505, -1.4472233057022095, 1.074123740196228, 0.2322501242160797, -1.6230723857879639, -1.2197738885879517, 1.0339144468307495, 1.175164818763733, -0.1147216260433197, -1.1574476957321167, 1.2280216217041016, -1.7371110916137695, -0.20565740764141083, 0.3973386287689209, -0.2576320469379425, 0.8533743619918823, -0.002763817785307765, -0.5666720867156982, -0.49828922748565674, -0.8749921917915344, -0.2499203085899353, 0.5997238159179688, -2.053452253341675, -0.03792329132556915, -0.9289017915725708, -1.0768016576766968, 0.08824439346790314, -1.3289769887924194, -0.24346813559532166, -2.0190749168395996, 0.8115473985671997, -0.8098349571228027, -0.3427610397338867, -1.6906461715698242, 0.8568019866943359, -0.4578184187412262, 0.20992206037044525, 0.45170918107032776, 0.07476472109556198, 0.0021603212226182222, 0.5828995108604431, 0.3144742250442505, -0.9089421033859253, -0.5436215996742249, 0.0728805735707283, 0.37589073181152344, -0.3471671938896179, 0.0002463296987116337, 0.2323005646467209, -0.02000380866229534, 0.8064242005348206, 1.1980509757995605, -0.2771230936050415, 0.6632777452468872, -0.4245564043521881, 0.6379727721214294, 2.30471134185791, 0.6493335962295532, 0.18881390988826752, 1.668030858039856, -1.5502898693084717, 0.7696030735969543, -0.34350138902664185, 1.5406266450881958, -0.9172903299331665, 1.0550134181976318, -0.3091640770435333, -2.2806475162506104, 0.3889733552932739, 0.09858494251966476, 1.7171825170516968, 0.5747323036193848, -0.43780750036239624, 0.8714553713798523, -0.06383353471755981, -1.1231449842453003, 0.27675288915634155, -1.5434720516204834, -0.16846194863319397, 0.3872617781162262, 2.6108014583587646, 1.3195242881774902, 1.3491088151931763, 0.6234363913536072, 2.176675796508789, 0.21684692800045013, -2.2476980686187744, 0.6626228094100952, -0.9691611528396606, 0.4847193658351898, 0.9300052523612976, 0.7130028605461121, -0.27380606532096863, -0.4298042058944702, 1.4197677373886108, -0.24709045886993408, -0.5354171991348267, 0.049678701907396317, 0.6174373030662537, -0.15113024413585663, 0.39432957768440247, -0.8664559721946716, 1.5804294347763062, 0.5521637201309204, 0.2388056367635727, -0.8829537034034729, -1.0039926767349243, 0.256869912147522, 0.8556504845619202, -0.10152047127485275, 1.3354332447052002, 1.1002923250198364, -1.151013970375061, 1.556693434715271, 0.4766468107700348, 2.9457271099090576, -2.2522952556610107, -0.11554593592882156, 1.2035998106002808, -1.4013906717300415, 0.8246974349021912, -0.5456617474555969, 0.05611608549952507, -1.1496069431304932, 2.0133259296417236, 0.2344581037759781, 1.8247382640838623, 0.060924071818590164, 1.4437713623046875, 0.6453313231468201, -0.4254974126815796, -0.4817110002040863, 3.1715807914733887, -1.6027411222457886, -0.1584310233592987, 1.9050036668777466, -0.06796421110630035, 0.4052361845970154, -0.4123367369174957, 0.21143001317977905, 0.9049262404441833, 2.2755069732666016, -0.1621958315372467, 0.3410763740539551, 0.27787327766418457, 1.495617389678955, -0.2406105250120163, 0.09593474864959717, 1.1546604633331299, 0.6230682730674744, -0.41142234206199646, 0.7680374979972839, 0.8703598380088806, -0.7502771019935608, 0.8918693661689758, -2.1325948238372803, 1.2743712663650513, -0.12205016613006592, -1.0745182037353516, -0.41792258620262146, -1.1193445920944214, 0.8717569708824158, -0.885439932346344, 0.6705895662307739, 0.8143162131309509, -1.6152384281158447, 0.35466140508651733, 0.9178332686424255, 0.8162596225738525, -0.19595646858215332, 0.39374107122421265, -0.7248747944831848, 0.47176510095596313, -1.1756540536880493, 0.22577086091041565, -0.6115697622299194, -0.8593339323997498, 0.6183297038078308, 1.4176684617996216, -2.417602777481079, -1.7588165998458862, 1.6721333265304565, -0.17916883528232574, -1.1202763319015503, -0.0400395467877388, 0.6753702163696289, -1.148067831993103, 0.24526835978031158, 1.4919357299804688, -0.9837754368782043, -1.7890844345092773, 0.25669655203819275, 1.240882396697998, 1.8425559997558594, -1.381508231163025, 0.5416386127471924, -0.23229274153709412, 0.09988527745008469, -0.4600524306297302, -0.1120421290397644, -0.8757621645927429, 0.9503535628318787, -0.7055743932723999, -0.2951017916202545, -0.5790797472000122, 2.0582354068756104, 1.4112917184829712, 1.4273093938827515, -1.129082441329956, -0.866335928440094, 0.2660832405090332, 1.1231942176818848, 0.35980910062789917, -1.4400404691696167, -0.1838134378194809, 0.6775668263435364, 0.8796982169151306, -0.3538494110107422, 2.065863847732544, -0.25965896248817444, -0.7033210396766663, -0.4282184839248657, -2.0313994884490967, 0.9365893602371216, 2.0107457637786865, -0.6799846887588501, 1.2753887176513672, 0.4958520531654358, 0.6784570217132568, -0.0944780483841896, -0.2853984534740448, -0.20898312330245972, -0.5637707114219666, -0.6669875383377075, -1.0797117948532104, 1.434090256690979, -2.089339017868042, -0.017565255984663963, -1.2922883033752441, -1.0379812717437744, 0.07641902565956116, 1.2762254476547241, 0.17189373075962067, -0.11913534998893738, -0.1903979480266571, -0.9377651810646057, 1.5589629411697388, -0.9644781947135925, 0.294798344373703, 0.44636157155036926, -0.8861690163612366, -1.8748531341552734, -0.039462532848119736, -0.9591381549835205, 0.43884652853012085, -0.3956034183502197, 0.3935006856918335, -1.091820240020752, -1.3933459520339966, 0.03873836621642113, -1.5679726600646973, -0.4892981946468353, -2.0059382915496826, -1.0458223819732666, 0.9890087842941284, 0.14247295260429382, -0.1858874261379242, 0.14428378641605377, 0.22320812940597534, 1.3812180757522583, -1.8104482889175415, 0.43579620122909546, -0.4692419469356537, -2.1687138080596924, -0.9991405010223389, -0.5116180777549744, -0.47566908597946167, 0.25826016068458557, -0.6448684334754944, 1.150177001953125, -0.4510664939880371, -0.5803084969520569, 1.252954125404358, 1.191771149635315, 1.4180078506469727, -1.8549884557724, 0.33308112621307373, 0.2992100119590759, 2.313817024230957, 0.8618778586387634, 0.7873417735099792, 0.3976989984512329, 1.9727942943572998, 0.41513702273368835, 0.8574045300483704, -0.6507558822631836, 1.130340337753296, -0.10727735608816147, 1.1620789766311646, -0.8696532249450684, -0.06831240653991699, -0.7102057933807373, 0.8468602299690247, -0.6918873190879822, -0.9660144448280334, 0.4255305826663971, 1.5425655841827393, 1.3555142879486084, -1.6199208498001099, 0.1501069813966751, 1.4637112617492676, -0.029734428972005844, -0.1563708484172821, 0.02011488378047943, -1.0629700422286987, 0.43783801794052124, -1.6167422533035278, 1.1121890544891357, -1.2012194395065308, -0.18428783118724823, 1.2865021228790283, -0.3879956603050232, 0.3345135450363159, -0.6346902847290039, -0.5177956819534302, 0.8350319266319275, 0.5301100015640259, -0.13428117334842682, -1.4821949005126953, 0.25861355662345886, -0.29822486639022827, -1.1938778162002563, -2.033921003341675, -0.044117555022239685, -0.697111964225769, -0.3225556015968323, -1.4052672386169434, 0.8534635901451111, 0.09536522626876831, 0.02181839570403099, 0.4010489583015442, -0.4914238452911377, -0.09401465207338333, -0.7546527981758118, 1.1274162530899048, -0.10257479548454285, 0.22153650224208832, 0.01263444870710373, -0.7327190041542053, 0.5427228212356567, 1.3743500709533691, 0.8417397141456604, 0.14905667304992676, -0.20739807188510895, 2.709646224975586, -0.27401062846183777, 0.6202887892723083, 0.6809156537055969, -0.15274234116077423, 0.5069068074226379, 0.5680323839187622, 0.9012717008590698, 1.1097551584243774, -1.1951086521148682, 1.3142321109771729, -1.2462575435638428, -0.6873963475227356, 1.2323564291000366, 0.7245184779167175, -1.305138111114502, -0.7377777695655823, -0.22357383370399475, -0.3506982922554016, -0.7259398102760315, 1.169701337814331, -1.6907434463500977, -1.6134140491485596, -1.2219805717468262, 0.9362057447433472, 0.28009936213493347, -1.221267580986023, 1.4301085472106934, -1.298319697380066, 1.366201639175415, -0.9154190421104431, 0.06269127130508423, -0.8286824226379395, 0.5057805776596069, 1.1368550062179565, -0.6423180103302002, 0.838188648223877, -1.2871843576431274, 1.8658710718154907, 2.0685057640075684, 2.0133230686187744, 0.5200856924057007, -0.6278949975967407, 0.3756446838378906, 0.7189429402351379, -0.5183747410774231, -0.8505481481552124, 1.2211579084396362, 3.085357189178467, -0.1112358570098877, -0.24350079894065857, -1.1865473985671997, 0.40489664673805237, -1.0903549194335938, 1.2251189947128296, -0.13770607113838196, 0.8337439298629761, -1.2551196813583374, -2.4366707801818848, 1.0684491395950317, 0.7525733113288879, -1.1787225008010864, -0.14294148981571198, -1.260509967803955, -0.8228943347930908, 0.39048826694488525, -0.4009869694709778, -0.17806687951087952, 1.3846449851989746, -0.23421339690685272, -1.818835973739624, 0.07315897941589355, 0.6135181188583374, 0.20137697458267212, -0.5572900176048279, 0.3797351121902466, -0.28581464290618896, 1.4158096313476562, 1.7564697265625, 0.035309746861457825, 0.30960914492607117, -1.0560166835784912, -1.0039020776748657, 0.6733735203742981, -0.07576484233140945, 1.304029941558838, 0.3812313973903656, -0.2371719777584076, 0.5224167704582214, 0.5714907646179199, 0.028719598427414894, 2.0233347415924072, -0.6662135124206543, 0.2839698791503906, 0.8493221998214722, 0.4851965606212616, 0.04262009263038635, -0.21710780262947083, 2.0246312618255615, 0.3614600896835327, 0.7573433518409729, 0.5372344255447388, 0.9450609087944031, -0.12641765177249908, 1.3967453241348267, -2.1816720962524414, 0.4609738886356354, -0.32714322209358215, 1.2099075317382812, 0.27658936381340027, -0.6195990443229675, -0.4215884208679199, -0.42267459630966187, 0.8777403235435486, -0.015914103016257286, 0.031756673008203506, 0.30437007546424866, 0.7981886863708496, -0.43331238627433777, 0.9531702399253845, -0.5377141237258911, -0.41029679775238037, 0.2569928467273712, 0.6698174476623535, -0.017886925488710403, 0.33615073561668396, -0.7122976779937744, 1.0388611555099487, -0.6084989905357361, -0.21768268942832947, -0.4232083857059479, 0.23387344181537628, 0.08245289325714111, -1.7563514709472656, -1.6301381587982178, 2.3182055950164795, 1.5084434747695923, 0.6009496450424194, 0.7161372303962708, 0.5799110531806946, 0.7268254160881042, -1.250956416130066, -1.7299509048461914, -0.8856406807899475, -1.1465257406234741, -0.6144717335700989, -0.5086162090301514, 0.5906339883804321, 0.4724242389202118, -1.0291846990585327, 0.38754111528396606, 0.789269745349884, -1.6858311891555786, -0.3939819633960724, -0.7272219061851501, 0.8914904594421387, -1.026858925819397, 2.1204686164855957, -0.09122856706380844, -0.26169145107269287, -1.8197293281555176, 0.3935486376285553, -0.8716797828674316, 1.3703324794769287, -0.7262919545173645, -0.09005536139011383, 1.452215552330017, 1.9299302101135254, -1.1269317865371704, 0.9166661500930786, 0.44148126244544983, -0.3510522246360779, -0.22795343399047852, -0.8731332421302795, -0.42321571707725525, -0.42033249139785767, 0.6223910450935364, 0.42653051018714905, -1.0841944217681885, 0.4596850574016571, 0.4143139719963074, -0.05296808481216431, 0.15054276585578918, -1.409264087677002, 0.5138366222381592, 1.395734190940857, 0.4779452681541443, 1.3302382230758667, 1.0989487171173096, -0.9846519827842712, 0.5870727300643921, -0.42681175470352173, 0.040245942771434784, 0.4627547264099121, 1.2211602926254272, 0.12902380526065826, 1.3571330308914185, -0.30422598123550415, -0.09648312628269196, -0.7428631782531738, 0.8041276931762695, -0.5198091864585876, -0.6552011966705322, 0.40485677123069763, 0.7531262040138245, -0.577174186706543, -0.14245472848415375, 0.1349964588880539, 0.9640133380889893, 0.21822045743465424, -1.204813838005066, 1.0041712522506714, -1.005125880241394, -0.6615751385688782, -0.04854077845811844, 0.313246488571167, -1.330296277999878, -0.08812753111124039, -0.2788102626800537, 0.6254292726516724, -0.4529092013835907, -0.38352495431900024, -2.1067416667938232, -1.6012269258499146, -1.4332013130187988, 0.39371609687805176, 0.19069236516952515, -0.8921685814857483, 0.16138246655464172, -0.5886008739471436, -1.2164711952209473, -0.08085494488477707, -1.8672971725463867, 0.09742839634418488, 0.6724879145622253, 0.4898422360420227, 1.6795551776885986, -0.41486987471580505, -0.851327121257782, -0.5770720839500427, -0.10604404658079147, 1.4793835878372192, 1.1685960292816162, 0.2887450158596039, 2.431864023208618, 1.9091335535049438, -1.4292453527450562, -0.7242852449417114, -1.0520119667053223, -0.9603237509727478, 1.191729187965393, 2.228102445602417, -0.2443913221359253, 1.4555420875549316, -1.4326545000076294, -0.4337809681892395, 0.45777419209480286, -1.8319730758666992, 2.0031397342681885, -0.9125243425369263, 0.4261704981327057, 0.4348071813583374, -0.0075371237471699715, -1.4783979654312134, 0.12085811048746109, -0.3457489311695099, 1.0984026193618774, -0.5567872524261475, 0.5860956311225891, -1.359492540359497, -0.24515004456043243, 1.0684198141098022, 0.5904262065887451, 1.7754175662994385, -0.6528531312942505, -0.23525702953338623, 2.2731518745422363, 1.0392173528671265, 0.7411455512046814, -1.6805365085601807, -2.0631449222564697, 0.5456898212432861, -0.12108641117811203, -0.7667142152786255, 0.2551279366016388, -0.14525337517261505, 0.9900009036064148, 0.006602352485060692, 0.5504807829856873, 1.4057788848876953, -0.060225971043109894, 0.5807485580444336, -0.5687307715415955, 0.27381232380867004, -0.3500944674015045, 2.2590904235839844, 0.4278596341609955, -0.15269821882247925, -0.10713603347539902, -1.0803459882736206, -0.09418819844722748, -0.31683456897735596, 0.7138350605964661, -1.1727632284164429, -0.5845569372177124, 0.22993868589401245, 0.7047704458236694, -0.5430166721343994, 0.8995795249938965, -1.2489240169525146, -1.9329105615615845, -1.1400936841964722, -0.040815599262714386, -0.5475295186042786, -1.232061505317688, 0.009091715328395367, 0.41528019309043884, 1.5437698364257812, -0.050011806190013885, -0.6817043423652649, 0.5324280261993408, -1.4517052173614502, -0.5279664397239685, 0.19725728034973145, 0.4431033730506897, 0.5034094452857971, 0.37506774067878723, -1.0869067907333374, -0.7074055671691895, -0.26156964898109436, -0.6837775707244873, 0.10145305842161179, -0.35685327649116516, 0.314056932926178, 0.2683218717575073, 1.2776323556900024, -1.9842941761016846, -0.5152995586395264, 0.012872759252786636, 0.09204728901386261, -0.017712589353322983, 0.9448232650756836, -0.7575748562812805, -0.25810807943344116, 1.0889633893966675, -0.7445722818374634, 1.01012122631073, -0.27070868015289307, 0.12130105495452881, -1.8014073371887207, 0.08706684410572052, 0.8831559419631958, -0.7664850950241089, 1.893244743347168, 0.6713650226593018, 0.3621988594532013, -1.1358085870742798, -1.1340652704238892, -0.2716364860534668, 0.149671733379364, 0.513699471950531, -1.6885969638824463, 0.18102380633354187, 0.2190895825624466, -0.3636092245578766, 0.08568469434976578, 0.23063233494758606, 1.7107023000717163, -0.7754246592521667, 0.2628350555896759, -1.0018303394317627, 0.4255770742893219, -0.12869544327259064, -0.05879245325922966, 1.8003078699111938, -0.7504212260246277, 0.1974705010652542, 0.9821633696556091, 1.2203923463821411, -0.5784286260604858, -0.4250153601169586, -1.4054808616638184, 1.3782062530517578, 0.09871821105480194, -0.5605954527854919, 0.45249250531196594, 1.2300094366073608, -0.20088157057762146, -1.653623342514038, 1.0823019742965698, 0.7517374157905579, 0.3114457428455353, 0.8065948486328125, -0.07623853534460068, -0.06762310862541199, -0.46617424488067627, -0.4278656542301178, -1.8049967288970947, -0.2678697407245636, 0.9892125129699707, 0.9130243062973022, 0.24892212450504303, -0.3560599386692047, 0.08647321164608002, 0.538606584072113, -0.27457138895988464, 1.1710227727890015, 2.9166574478149414, -0.4195425510406494, -1.9050841331481934, 0.7648617029190063, 0.021052291616797447, -2.4296422004699707, 0.906330943107605, 1.062568187713623, -1.213005542755127, 0.404470831155777, 0.22630150616168976, -1.8743168115615845, -0.44241657853126526, -1.9297645092010498, 0.8535995483398438, 0.7283773422241211, 1.3436243534088135, 0.505467414855957, -0.3432127833366394, -0.8110762238502502, -0.2651295065879822, 0.7617188692092896, -0.6306428909301758, -0.14749900996685028, -0.10454156994819641, 0.6730200052261353, 0.5833919048309326, -0.08355245739221573, -1.3329607248306274, 1.063830852508545, -0.5377960205078125, 1.0597989559173584, -2.0678160190582275, 1.3260149955749512, -0.9528155326843262, 0.9254531860351562, -0.6558050513267517, -2.065962076187134, 0.18335698544979095, 0.281421422958374, 1.3445661067962646, -1.398160457611084, -0.14606519043445587, -0.36534368991851807, 1.145667314529419, -1.570643424987793, 1.104407548904419, -0.9267104864120483, 0.015805015340447426, 0.6234449744224548, -0.19538599252700806, 1.4283970594406128, -1.475468635559082, -1.2158312797546387, -1.6177725791931152, 0.8996015787124634, 1.602002501487732, 0.06691008061170578, -1.1460200548171997, -1.2883855104446411, 0.04519473761320114, 2.043088436126709, 1.7298126220703125, -0.784934401512146, -0.0948784202337265, -2.0094857215881348, 0.020980434492230415, -0.681344747543335, -1.1230876445770264, 0.6949219107627869, 0.9400679469108582, -0.3471938371658325, -1.0341615676879883, 0.705012857913971, 1.7915904521942139, 0.7531226873397827, 1.7403415441513062, -1.4568183422088623, 0.11832477152347565, -1.5943198204040527, -1.060746192932129, -1.0960508584976196, 0.1907966285943985, -0.3675747513771057, -2.797344446182251, 0.3922123610973358, 1.1116830110549927, -1.0760947465896606, 0.8577172756195068, 0.6287493109703064, 0.19858014583587646, -0.5414209961891174, -0.7507850527763367, -1.7015018463134766, 1.33659827709198, 0.405926376581192, -0.984329104423523, -1.1652485132217407, 0.9403724670410156, 0.47110286355018616, -1.2687994241714478, 0.17691820859909058, 3.5069189071655273, 1.4170262813568115, -0.6911598443984985, 0.09535157680511475, 0.3296091556549072, -0.17485113441944122, 0.33446282148361206, -0.8267812132835388, -0.18093055486679077, -1.700804352760315, 0.3349778354167938, 0.352186918258667, 0.46111613512039185, -0.4402998089790344, -0.37893807888031006, 1.3282418251037598, 0.9280694723129272, 0.7902170419692993, 0.8249227404594421, -0.28670743107795715, 1.9114902019500732, -0.8930843472480774, 0.2065357267856598, 0.48244842886924744, 1.0720163583755493, -1.270397424697876, 0.012276571244001389, -0.23582705855369568, 0.3604709208011627, -0.21375757455825806, -1.3512040376663208, 0.2155947983264923, -0.6554626226425171, 1.0008286237716675, -0.7281590700149536, -0.537466824054718, -0.12355158478021622, -0.8222641348838806, 0.7599027752876282, 0.2873472571372986, -0.5335747003555298, -0.25390979647636414, -0.43521666526794434, 0.2146076112985611, 1.7679027318954468, -0.013333740644156933, 0.39676710963249207, 0.27313703298568726, 0.8598540425300598, 0.07579250633716583, -0.21658797562122345, 0.13244205713272095, 1.0679205656051636, -0.6852284073829651, -0.33027657866477966, 0.20599769055843353, 0.7229704856872559, -0.7374448180198669, -1.7427091598510742, 0.635136604309082, -0.5679177641868591, 0.9330067038536072, -1.3181711435317993, -1.5885850191116333], [1.1265949010849, 0.6773852705955505, -1.4114638566970825, 1.2303739786148071, -0.16055350005626678, -0.14680299162864685, 0.8987374901771545, 0.8313964009284973, -0.7216268181800842, -0.5817307233810425, -1.254411220550537, -0.5355648994445801, -0.8209362030029297, -0.32455897331237793, 0.9430809020996094, 0.017956644296646118, -0.2701438367366791, 0.5281031131744385, -1.9494755268096924, 0.259079247713089, 0.48157015442848206, -0.3807384967803955, -0.3024669885635376, 0.7948559522628784, 0.8487347364425659, 0.31899115443229675, -0.6391135454177856, -1.4727452993392944, 0.9988973736763, -1.1944501399993896, -0.6099940538406372, 0.15076518058776855, -1.2058273553848267, 0.5943368077278137, -0.9740561246871948, 0.05412520095705986, 1.2636295557022095, -1.470763087272644, -0.8727453351020813, -1.5647894144058228, -0.19859522581100464, -0.8127398490905762, -0.6998307108879089, 0.3229566216468811, 1.0132471323013306, -0.40896114706993103, 0.4112159013748169, 0.2827553153038025, -1.2732349634170532, -0.7505242824554443, -0.10201236605644226, -0.5522918105125427, -1.2855916023254395, -1.3613065481185913, -1.4478906393051147, 3.314551591873169, -0.05015961453318596, -1.8894542455673218, 0.08092642575502396, 0.5224117636680603, 1.0416251420974731, -0.9478099942207336, -0.353512167930603, 1.9399796724319458, -0.003557613119482994, 1.2278053760528564, 0.5880475640296936, -0.6666213274002075, 0.8210320472717285, -0.05291552469134331, -1.4843860864639282, -1.648898959159851, -0.4937356114387512, -0.10547620803117752, -0.7015930414199829, -1.0953930616378784, 0.0875975713133812, -0.14981137216091156, 1.109074592590332, -2.1690545082092285, -0.2998150885105133, 0.6888815760612488, -0.8211880922317505, 0.9951485395431519, 0.3367724120616913, -0.9767270088195801, 0.05262095481157303, 0.46821191906929016, -1.2260292768478394, -0.4085146486759186, 0.473764568567276, -0.6822310090065002, 0.02118665724992752, 1.8074487447738647, 0.6968638896942139, 1.8713269233703613, 0.8384644389152527, 2.5897653102874756, -1.232651948928833, 0.5575535297393799, -2.207129716873169, -0.4392692744731903, 1.4464895725250244, -0.7078545689582825, -0.4933365285396576, -0.8461772799491882, 0.22771112620830536, -0.837319552898407, 0.9842556715011597, -1.9022504091262817, 0.41715821623802185, -0.3821084797382355, 1.185361385345459, -0.8631083965301514, -0.642566442489624, 0.3620690703392029, -1.22591233253479, -0.7399513125419617, -0.6152341961860657, -0.23997288942337036, -0.9997999668121338, 0.8454433679580688, -1.0597537755966187, 0.3644885718822479, 0.5246738195419312, 0.7688171863555908, 1.3117660284042358, 0.2704715132713318, 2.235330581665039, -0.809869647026062, 1.0475859642028809, 0.3015524744987488, 0.3679642975330353, -1.1713935136795044, 1.1119803190231323, 0.3844742476940155, 0.23944884538650513, 0.49962538480758667, -1.5331268310546875, -0.2048858404159546, 1.1011922359466553, 0.12178490310907364, 0.19236744940280914, -1.1144522428512573, -0.33858081698417664, -0.5484815835952759, -0.6170358061790466, -1.78031325340271, 1.8814955949783325, -1.8557028770446777, 0.03398711979389191, 2.292820930480957, 1.2232427597045898, 0.012900102883577347, -0.8983988761901855, -2.0354747772216797, 0.7239698767662048, -0.3099329471588135, 0.6126642227172852, -1.2173261642456055, 0.2536199688911438, -0.38135388493537903, -1.4059885740280151, 0.7978489995002747, 0.5959535837173462, -2.5870156288146973, -0.45444318652153015, -1.0420043468475342, -0.9875633120536804, -1.3866642713546753, 0.08911602199077606, -0.5429373979568481, -1.1516196727752686, -2.552929401397705, 0.13539497554302216, 0.38542625308036804, -1.008301854133606, 0.33936211466789246, 1.392148733139038, -0.36749181151390076, -1.704683780670166, -0.43119949102401733, -0.2553514242172241, -1.367862343788147, -1.2669613361358643, 0.3141569495201111, -0.012059459462761879, -0.0022680661641061306, 3.4775192737579346, 0.5899435877799988, 0.4385421574115753, 0.3554503321647644, -0.5755277276039124, -0.5087956190109253, 1.024377703666687, 0.5519492030143738, 1.508781909942627, 1.018354892730713, 0.8854455351829529, -0.8457567095756531, -0.05216463282704353, 1.6906418800354004, 1.0460866689682007, 0.054881103336811066, -1.165714144706726, 0.9561125636100769, -1.8296091556549072, 1.3373417854309082, 0.9075923562049866, -0.3675674498081207, -0.16833217442035675, 1.1281970739364624, -0.00020485161803662777, 1.3851312398910522, -0.4413990080356598, -2.3525078296661377, 0.02636067569255829, -1.9111343622207642, 0.6781375408172607, 1.0809491872787476, -1.8029431104660034, -0.6908183097839355, -0.31332263350486755, -0.05991274490952492, 2.7712886333465576, 0.024676170200109482, 1.551283597946167, -0.40105876326560974, 0.05570061504840851, -0.2927200496196747, -0.001174276345409453, -0.33189892768859863, -0.4415472149848938, 0.1624504178762436, -0.3667677342891693, -1.2199512720108032, 0.4318375289440155, 1.4351961612701416, -0.35865405201911926, -0.5030109882354736, -1.869941234588623, 0.059361860156059265, 0.7311834096908569, -0.2669110894203186, -1.0995795726776123, 1.2079638242721558, -0.4893098473548889, 0.43891406059265137, 0.16080714762210846, -0.9604425430297852, 1.3774068355560303, 0.899632453918457, 0.29931092262268066, 0.4891394376754761, -0.5418305993080139, 1.0620876550674438, -1.8606537580490112, -1.590322732925415, -1.915436029434204, -0.7547125220298767, 0.15725195407867432, 1.025254726409912, -0.04828564077615738, -0.05423348769545555, -1.098070502281189, 0.22367653250694275, -1.9218239784240723, 0.8095311522483826, 0.6586476564407349, -0.08059868216514587, -0.4627336263656616, -0.33222290873527527, 0.1521202176809311, 0.3291640281677246, -0.5584923624992371, 1.1928513050079346, -0.29779914021492004, 0.7957789897918701, 0.468837708234787, -0.3004768490791321, 0.8283847570419312, 1.1678327322006226, -0.8104465007781982, -1.2925199270248413, -0.23094157874584198, -0.8116306066513062, -0.2436397224664688, -0.8355361819267273, -0.966214120388031, 0.22284096479415894, -0.07745125889778137, 1.206691026687622, -0.36263200640678406, 0.47932741045951843, -2.056709051132202, -0.004966926295310259, 1.3763288259506226, 1.2436244487762451, -0.9271321892738342, 2.0133063793182373, 0.6593163013458252, -1.1597176790237427, 2.2025654315948486, -0.5814775228500366, -1.609639048576355, 0.19943106174468994, 0.41018325090408325, -0.8195740580558777, -0.5672808289527893, -0.02122299000620842, -1.5576059818267822, 0.9068591594696045, -0.24357207119464874, 0.3563498556613922, -1.7100815773010254, 0.47914794087409973, -0.270226389169693, -0.4904637634754181, -0.3720497786998749, -0.935200572013855, 0.9813075661659241, -1.0193791389465332, -1.3481065034866333, -0.2980421185493469, -0.37125828862190247, -0.4645942747592926, 1.0126909017562866, 1.1009336709976196, 1.2624859809875488, 1.6371749639511108, -0.9733746647834778, -1.3571432828903198, -2.4088101387023926, 0.27331677079200745, -1.302653431892395, -0.18237896263599396, 1.0225181579589844, 0.37540990114212036, -0.3933262825012207, -0.5044859647750854, 0.3861393332481384, -0.8106766939163208, 0.00753095094114542, 0.1052016168832779, -0.9133986830711365, -1.0599253177642822, -0.47859296202659607, 0.7670695781707764, -1.276434302330017, -0.16392602026462555, 1.3240450620651245, 0.08793464303016663, 0.19114108383655548, -1.2442524433135986, -0.009109942242503166, -0.478042334318161, 0.6482420563697815, 0.8075128197669983, -0.7976988554000854, -0.7287344336509705, -0.7694771885871887, -0.6198151707649231, 0.07933513075113297, 0.2773846685886383, -0.8982961177825928, -1.462437629699707, 0.64588463306427, -0.9252104163169861, 0.0603940449655056, -0.4093112051486969, 1.072439432144165, -0.2902458608150482, 0.3333432376384735, -0.8512949347496033, -0.5671632885932922, 1.30763840675354, 1.447701096534729, -0.19286826252937317, 1.1293690204620361, -1.0631837844848633, 1.4473083019256592, 0.03559911251068115, -1.6551718711853027, 0.2683354318141937, -1.0915813446044922, 0.7157256603240967, -0.415599524974823, -1.81387197971344, 0.6889159083366394, 0.10439343750476837, -0.5316356420516968, 0.18038935959339142, -0.2730843722820282, -1.054580807685852, 0.6174759864807129, 1.2480006217956543, 0.10599290579557419, -1.0544191598892212, -0.8056038618087769, 1.0515949726104736, 0.615036129951477, -1.5062458515167236, -0.02029426023364067, -2.329799175262451, 0.9909008145332336, 0.4356178045272827, 1.205316424369812, -2.167954444885254, 0.5966526865959167, -0.7209222912788391, 0.5115347504615784, -0.6387358903884888, -0.5839744806289673, -0.10561643540859222, 0.7135763764381409, -0.04721945896744728, 1.960924506187439, 0.3379923701286316, 2.5379786491394043, -0.5658396482467651, -0.3754514157772064, 1.7558627128601074, 2.057192087173462, 0.9728251695632935, -0.342436283826828, -1.4229652881622314, -1.3781615495681763, 0.9589065909385681, 1.4806965589523315, -1.6023497581481934, -0.887083888053894, -0.11474060267210007, -1.0378493070602417, 1.3348708152770996, -0.9168674945831299, -0.2038431316614151, -1.1160576343536377, -0.6016606092453003, 0.15070118010044098, -0.5668573975563049, -1.33173668384552, 0.005300948861986399, 0.7758788466453552, 0.9213478565216064, 1.0346455574035645, -1.59811270236969, -1.2875980138778687, 0.8424734473228455, 0.24982720613479614, -0.18286052346229553, 0.4671158790588379, 0.0779561921954155, -0.16157680749893188, -0.7761077880859375, 0.11931561678647995, -0.5988182425498962, -0.5222029685974121, -0.930957555770874, 1.2078375816345215, -0.35225239396095276, 1.0732041597366333, 0.7820364832878113, 1.0605045557022095, -0.3215423822402954, -0.17755910754203796, 1.2103840112686157, 0.6983553171157837, -0.3000684678554535, 0.8773247003555298, 1.0883605480194092, 0.6720433235168457, 0.43910977244377136, 1.7360094785690308, -1.6414982080459595, -1.3805686235427856, -0.21124263107776642, -2.17519211769104, -1.0089311599731445, -1.9578831195831299, -0.6728925704956055, 0.8497449159622192, -0.1986534148454666, 0.8151723146438599, -0.31181269884109497, -0.5180336833000183, -0.2200177013874054, -0.8215221762657166, 0.06046713516116142, 0.4321553111076355, -0.7640186548233032, -0.20554305613040924, -0.8774147033691406, 0.20714232325553894, -1.9547168016433716, 0.32115259766578674, 0.36482805013656616, -1.4189361333847046, 0.7569377422332764, -0.0033186725340783596, -0.34168770909309387, 0.08772844821214676, -0.694397509098053, -0.9850842952728271, 0.6833009123802185, -0.6821570992469788, -1.1672725677490234, -1.3388444185256958, -0.7647024989128113, 0.03035855107009411, -1.7795746326446533, 1.5924410820007324, 0.3721942901611328, 1.7626209259033203, -0.5359163284301758, -1.3727751970291138, 0.5830485224723816, 0.5463026762008667, 2.0041542053222656, 0.14931078255176544, -0.9983338117599487, -0.023294970393180847, -0.2471836358308792, -1.013075351715088, -0.3179838955402374, 0.19511450827121735, -1.526961326599121, 0.6785149574279785, -0.015211833640933037, 1.470438003540039, 0.16207225620746613, -1.4399489164352417, 1.8231929540634155, 1.2347328662872314, 2.134284257888794, -1.6729010343551636, -0.14727665483951569, 0.19198615849018097, 2.352372169494629, -0.3751501441001892, 1.0535385608673096, -0.3155200481414795, -2.0614941120147705, -0.16086333990097046, 0.6426088213920593, -1.0046906471252441, 0.5964834690093994, -0.9086485505104065, 1.9833518266677856, -0.43818774819374084, 0.12740792334079742, -0.8183324337005615, -0.8159006834030151, -1.8833861351013184, 0.8395440578460693, -0.6811133027076721, -0.28469204902648926, -0.2559659779071808, 1.2254705429077148, -1.8285095691680908, -0.3225288987159729, 0.2800854444503784, 0.41029220819473267, 0.5074913501739502, 0.07647224515676498, 0.8628820180892944, 0.9996775984764099, -0.49755722284317017, -0.38987383246421814, -0.7861946225166321, -0.6777511239051819, 1.0735139846801758, 0.16934476792812347, 0.3173450231552124, -0.49950844049453735, -0.24637751281261444, -0.7045966982841492, -0.34904056787490845, -0.19649140536785126, -0.7961335182189941, 0.4089869260787964, 1.025217056274414, 0.5235758423805237, -0.2788809835910797, -0.8275764584541321, 2.1053950786590576, 0.10697269439697266, 0.1448030322790146, -0.17973384261131287, -2.2046053409576416, -1.0703012943267822, 0.3408682644367218, 0.19527898728847504, -3.9417264461517334, -0.762529194355011, 0.8167065978050232, 1.299180507659912, 0.296872079372406, 0.5612459778785706, -0.20848223567008972, 0.35854727029800415, -0.394048273563385, 1.0333588123321533, -1.1639364957809448, 1.0815672874450684, 0.3225826621055603, -0.5547865629196167, 0.299656480550766, -0.5690966248512268, -0.11331340670585632, -1.4960942268371582, -2.989988088607788, 0.3701959550380707, 1.3658878803253174, 0.9380344152450562, -0.5181884169578552, 2.437601089477539, 0.6469343304634094, -0.2801569998264313, -0.3040703535079956, -0.5252724885940552, -2.5808088779449463, -1.183363437652588, -0.24064281582832336, -1.2884647846221924, -0.6654102206230164, 1.732083797454834, 0.6530400514602661, -0.14967429637908936, -1.9403867721557617, -0.548082172870636, -0.915932834148407, -1.2573790550231934, -0.565004289150238, 1.0088279247283936, 0.6548786759376526, 1.001859426498413, -0.9783706068992615, -0.7492081522941589, -1.9287089109420776, 0.4667046368122101, -0.9713057279586792, -0.7008373141288757, 0.3982684016227722, 1.1317791938781738, 0.6074302792549133, -0.22568657994270325, 0.6207423806190491, -0.5055540800094604, -0.027652086690068245, 0.9354739785194397, 0.06126080080866814, 0.2582920789718628, -1.0806814432144165, 1.4465522766113281, -0.23561877012252808, -1.0954464673995972, -1.216052770614624, 2.1101293563842773, -2.241219997406006, -1.5779088735580444, -1.3121870756149292, -0.014401653781533241, 0.8019577264785767, 0.3384172320365906, 1.207872986793518, -1.520703673362732, 0.3164358139038086, -0.5429131388664246, -1.1659643650054932, -2.1169915199279785, 1.1534359455108643, 0.8375844955444336, -0.5337769389152527, -2.433689594268799, -0.6867027878761292, -1.5369380712509155, -0.6434465050697327, 0.8114844560623169, 1.705610990524292, 0.9245033264160156, 1.5376598834991455, -1.6284750699996948, 0.15465638041496277, -1.0312782526016235, -0.4162687063217163, -0.6576592326164246, 0.5564330220222473, 0.33037739992141724, 0.7035309672355652, -0.06272052973508835, -1.0268205404281616, -0.9613969922065735, 0.8246913552284241, -0.813417375087738, -0.9077355861663818, 0.17118269205093384, 0.8152859807014465, -0.5598260760307312, 1.4357177019119263, 0.8032408952713013, -0.7394770979881287, -0.01998225599527359, -0.5251418352127075, 0.5279674530029297, 0.400209903717041, -0.6588603258132935, -0.7365940809249878, -0.738003671169281, 0.1091136783361435, 0.9442448019981384, -2.130350112915039, -0.2607039213180542, 1.0712010860443115, 1.4800560474395752, 0.47413745522499084, 1.681593894958496, -0.1527242511510849, -0.18731245398521423, 1.1322321891784668, -1.592658281326294, 0.11954629421234131, -1.0670822858810425, -1.6771790981292725, -0.40923595428466797, 0.40048888325691223, -1.8331016302108765, 1.6209126710891724, 0.28947713971138, 0.34119921922683716, -1.3138911724090576, 1.7509042024612427, -0.24021096527576447, -0.12601736187934875, 1.089479684829712, 0.5133466720581055, -0.05570747330784798, 0.9173180460929871, -0.11094480007886887, 0.4560500979423523, -0.18608708679676056, -0.022329572588205338, 0.6579671502113342, 1.3059748411178589, 0.6792182922363281, 0.10860326886177063, 0.1667311042547226, 1.319706678390503, 1.5186702013015747, 0.8141109943389893, 1.322648525238037, -2.475198268890381, -0.11013251543045044, 0.16349267959594727, 1.23941969871521, -0.23751811683177948, -0.45243334770202637, -0.3110741078853607, 0.34799930453300476, 1.4531437158584595, 1.3500874042510986, 0.5677063465118408, 0.6471205949783325, -1.251816749572754, 1.4031304121017456, 1.795277714729309, -1.2125240564346313, -0.18893025815486908, -0.46321120858192444, -0.5399172902107239, 0.9066296815872192, 0.5812286138534546, -0.39129385352134705, 1.038703441619873, -0.7791559100151062, -1.01738703250885, -0.8202853798866272, -0.6253982186317444, 0.30498966574668884, 0.5459631681442261, -0.029466642066836357, 0.2968064546585083, -0.27076518535614014, 0.715671181678772, -0.7863120436668396, -0.8980938792228699, -0.05155669525265694, 0.29193493723869324, -2.282212734222412, -0.5997687578201294, 0.10069962590932846, 0.815466582775116, 2.7886292934417725, 0.13467971980571747, -0.5187853574752808, -1.2702453136444092, 0.6607133746147156, -0.6738908886909485, 0.6167700886726379, 2.0110440254211426, -0.6363956928253174, -0.8659022450447083, -0.34865134954452515, 0.43584388494491577, -0.5058154463768005, 1.2521307468414307, -0.1207861602306366, -2.8237643241882324, -1.3713045120239258, -1.1588311195373535, 0.24065609276294708, -0.1359935998916626, 0.07550492882728577, -0.5194116830825806, 2.31683087348938, 0.010599438101053238, 0.08633929491043091, -2.0655171871185303, 0.5257850289344788, 0.04284758120775223, 2.51676607131958, 0.22957512736320496, 0.3252515494823456, -0.3856547176837921, -0.8682779669761658, 0.31288084387779236, 1.4632688760757446, -0.39277347922325134, -0.120263010263443, -0.8993805646896362, 0.8204094171524048, 1.7212560176849365, -2.805511474609375, 0.7730149626731873, 0.5316179394721985, -0.39375123381614685, 0.7160950899124146, 0.4611329436302185, -0.2869282364845276, -1.0936172008514404, 0.18952016532421112, 0.3936607837677002, -0.36266374588012695, -0.14439287781715393, -0.840894877910614, -1.1477049589157104, 0.49582380056381226, -1.2445389032363892, 0.5265299081802368, -0.41529300808906555, 0.2550593614578247, -0.5692185759544373, 0.37147819995880127, -0.451124906539917, -0.025450821965932846, 0.17908555269241333, -0.47431889176368713, 1.5724525451660156, 0.2923993170261383, 1.5783069133758545, 0.6070401668548584, -1.229655385017395, -0.587776780128479, -0.9306118488311768, 0.3480328321456909, 0.2044478952884674, -1.170996069908142, -0.6171274185180664, 0.2742941677570343, 1.0548474788665771, 1.8072080612182617, -0.27435633540153503, 0.7382768392562866, 0.9813635349273682, 1.615124225616455, 0.8644149303436279, 0.7263463139533997, 0.5231539011001587, 0.43112269043922424, -0.6703877449035645, 1.7768903970718384, -1.1233570575714111, 0.6370123624801636, 0.0487501285970211, 1.0657272338867188, 1.6281405687332153, 0.19126799702644348, 1.4170600175857544, 0.6034857630729675, 0.16761063039302826, 0.5296587944030762, -0.49507802724838257, 0.9345020651817322, -0.16040606796741486, -0.0783495083451271, 0.7578739523887634, -1.5181406736373901, 0.824495792388916, 1.231927752494812, 1.1358153820037842, 0.5812303423881531, -0.1892058253288269, -0.6651778817176819, -1.0509257316589355, -1.2967510223388672, -0.30189892649650574, -0.12265133857727051, 0.20210234820842743, -0.2866732180118561, 0.24383674561977386, -1.536268949508667, 0.3175872266292572, -0.1137208491563797, -0.6213630437850952, 0.6914432644844055, 2.6119606494903564, -1.065409541130066, 0.7644529938697815, 1.6474531888961792, -0.888802707195282, 1.7754048109054565, -0.6747750639915466, 0.79288649559021, 0.48021867871284485, 0.6491150856018066, 1.5005720853805542, -0.577541708946228, -0.6076215505599976, 1.7633320093154907, -0.20185969769954681, 0.8660876750946045, 1.560718297958374, 0.8536580204963684, -1.369642972946167, 1.1390113830566406, 0.2578737735748291, 0.17472602427005768, -0.7264043092727661, -0.6219442486763, -1.4026074409484863, -0.26059871912002563, -1.0712859630584717, -0.10980764031410217, 0.8348128199577332, -1.017056941986084, -0.18215665221214294, 0.27502045035362244, -0.2677459120750427, -1.546945333480835, -0.713934063911438, -1.202913761138916, -1.0791847705841064, -1.384971261024475, -1.5672547817230225, 1.8492279052734375, 0.8056323528289795, -1.0847387313842773, 0.1505875587463379, 0.12602776288986206, -0.12178187072277069, -0.3147313594818115, -0.3928523361682892, 0.12703543901443481, -0.5690796375274658, 0.1328953206539154, 0.0016776666743680835, -0.08021513372659683, 0.5215602517127991, 0.8703228831291199, -0.7264575958251953, -0.2161967009305954, 0.22647546231746674, 1.600799322128296, -1.461826205253601, 0.6092944741249084, 0.5481595993041992, -1.1278351545333862, 0.010620140470564365, 0.8522946834564209, -1.2750762701034546, -1.1046814918518066, -0.14365381002426147, 0.8828790783882141, -0.19772884249687195, -1.4963325262069702, 0.3298722207546234, -1.352323293685913, 0.018247509375214577, -0.08310719579458237, -0.6483538150787354, -0.8858761191368103, -0.02520902454853058, -0.49056777358055115, -0.4174613654613495, 0.261395663022995, -0.30434322357177734, -0.5340141654014587, -0.1059577614068985, 1.3296180963516235, 0.9404935240745544, -0.5024590492248535, -1.6118935346603394, -0.7241289019584656, 0.8918070793151855, -0.3565850257873535, 1.083390712738037, 1.4937037229537964, -0.9435309767723083, -1.2845995426177979, -0.7535061836242676, -0.6609402298927307, -0.26475003361701965, -0.5427501797676086, -0.1409863978624344, 1.1426976919174194, 0.8408322334289551, -0.839108407497406, -0.8701607584953308, -0.5817247033119202, -0.8942583203315735, -0.7757835388183594, 0.5565760135650635, 1.5862932205200195, 0.43345779180526733, 0.6943814754486084], [-0.45753633975982666, 0.567642092704773, 1.0113565921783447, -0.21485686302185059, 0.957199215888977, -0.8543238639831543, -1.9119981527328491, -0.33850881457328796, -1.5538991689682007, -1.133241891860962, 0.14102303981781006, 0.25916871428489685, 0.6678891181945801, -0.2252219319343567, -0.7463452219963074, -0.3100827932357788, 0.9376358389854431, 0.39647427201271057, -0.8516561388969421, -0.10989509522914886, -0.5515649318695068, -0.6026700735092163, -0.017030274495482445, 0.22603176534175873, 1.6811784505844116, -0.3492051959037781, -1.1224868297576904, 0.24063612520694733, 1.0705900192260742, -1.149261713027954, 0.4169032573699951, 0.18001237511634827, -0.6081704497337341, 0.9669070839881897, 0.5536571145057678, -0.8440560698509216, 2.5006954669952393, 2.3026931285858154, -1.8379017114639282, -0.926619291305542, -1.259717345237732, -0.18573744595050812, -0.5478231310844421, 1.0976015329360962, -1.0334439277648926, -0.4759308397769928, 0.09904635697603226, 0.6622155904769897, -0.016812818124890327, -0.3462367355823517, 0.879253625869751, 1.0130224227905273, 0.5936284065246582, -0.5640522241592407, 0.7318360209465027, 0.14543397724628448, -1.0984481573104858, -0.9700680375099182, -0.14116813242435455, 1.4190067052841187, 0.684053897857666, 0.5466746687889099, 0.12052380293607712, -1.5096628665924072, 0.9915464520454407, 0.6663830280303955, -0.16254447400569916, -0.7425782084465027, 0.8850148320198059, -0.7212435603141785, 0.5169010758399963, -0.2767864167690277, -1.5016098022460938, 1.1999735832214355, 1.0268632173538208, 0.23714683949947357, 1.5676106214523315, 0.6009193658828735, 0.8174161911010742, -0.4797564744949341, -0.272766649723053, 0.5050348043441772, 0.8951457738876343, 1.0162009000778198, 1.4689432382583618, 0.7575250864028931, -0.36745667457580566, -0.12213144451379776, -0.798575758934021, -0.11097823083400726, 0.5689986348152161, 0.7032851576805115, 2.572753429412842, -0.01400548592209816, -0.6059634685516357, 0.42143547534942627, -0.002374125411733985, 1.5170578956604004, -0.620582640171051, -1.498740315437317, -0.8265515565872192, 0.591180682182312, 0.4321589171886444, -0.10969235002994537, -2.4129157066345215, -1.1019210815429688, -0.5007523894309998, 0.6523454189300537, 0.1177402064204216, 0.7842950224876404, -2.7008614540100098, -0.43164193630218506, 0.05234415456652641, 1.8711891174316406, -0.920669674873352, 1.4772194623947144, 0.7597049474716187, -0.5315094590187073, 1.0250941514968872, 0.4860464930534363, -0.6925718188285828, 0.8777755498886108, -0.9195811152458191, -0.46887198090553284, -0.8752984404563904, -1.0103037357330322, -0.2627063989639282, -0.9074541926383972, 2.3785252571105957, 1.1645114421844482, 1.098249077796936, 0.4584091901779175, 0.285051167011261, 0.1525362730026245, 0.9457376003265381, -0.12802833318710327, 1.1465998888015747, 0.4448580741882324, 1.227277159690857, -0.6352022290229797, -0.7950248122215271, 0.20276705920696259, 1.3517814874649048, -0.021290790289640427, -0.4636913239955902, -0.5874460339546204, 0.21060210466384888, -0.8723022937774658, 0.402435839176178, 0.9899473190307617, -0.3976108431816101, -1.5602418184280396, -0.8479227423667908, 0.5023593902587891, -1.2364301681518555, 0.6024378538131714, -1.5485891103744507, 0.4604203701019287, 2.9168028831481934, -0.4680769443511963, -0.921387791633606, 1.4337173700332642, 0.23617370426654816, -1.0645248889923096, -2.023676872253418, 0.1762591153383255, 0.023597870022058487, -0.3387654423713684, -1.1717263460159302, 1.1572206020355225, -0.6838498711585999, 0.6321937441825867, 0.8589422106742859, 2.2138097286224365, 1.6560639142990112, -0.7393895983695984, -0.06139359623193741, 0.4942076504230499, -1.2245864868164062, 0.21430784463882446, 0.9210447072982788, 0.7411797642707825, -0.0360211506485939, 1.1179659366607666, -0.32783687114715576, 0.031163079664111137, 0.9145876169204712, -0.5432671308517456, 1.286664366722107, -1.0569539070129395, 0.32818278670310974, 1.3938205242156982, -0.2854333519935608, 0.07229261100292206, 0.8632668256759644, 0.46417078375816345, -0.4050857126712799, 1.1775325536727905, 1.3235527276992798, -0.12831953167915344, 2.9089062213897705, -1.1904174089431763, -0.9209783673286438, 0.07602579146623611, -0.2846405804157257, -0.5725393891334534, -0.18944011628627777, 0.5472589135169983, 1.183689832687378, -1.4472832679748535, 1.5362396240234375, 1.757828712463379, 1.1646201610565186, 0.06593654304742813, -0.9574833512306213, -0.4210564196109772, -0.10437874495983124, -1.8233851194381714, 1.1962897777557373, -0.9711377620697021, -0.22013334929943085, -0.14814472198486328, -0.6825818419456482, -1.7249749898910522, -0.7325997948646545, -0.6078451871871948, -0.1903112828731537, -1.2956677675247192, 0.3858262598514557, -0.14417079091072083, 1.1296519041061401, 1.2558157444000244, -1.330956220626831, -0.5027572512626648, 0.6069393157958984, 0.8261592984199524, -0.1998540163040161, -0.30685386061668396, -0.6094350218772888, -1.2254841327667236, 0.9937227368354797, 0.8355220556259155, -0.6205859184265137, 0.048185523599386215, -1.5028880834579468, 0.1158640906214714, 1.186988115310669, -0.789110004901886, 0.35076192021369934, -0.8184501528739929, -0.25831639766693115, -1.8677031993865967, 0.4962652325630188, 0.12014094740152359, 2.041092872619629, -0.6640292406082153, 0.9676697850227356, -0.17596657574176788, -1.0610105991363525, -2.806335926055908, 0.4102027714252472, -0.9524186253547668, 0.4797283411026001, 1.3203967809677124, 0.6710425615310669, -0.2141944319009781, 0.6184651255607605, 0.7012196183204651, -0.1490485668182373, 1.2050197124481201, 1.9886811971664429, 0.7333918809890747, 0.05140656605362892, -0.05568695068359375, -0.7286620140075684, -1.3074140548706055, -1.4417948722839355, 1.1529053449630737, -1.1572086811065674, 0.3154418170452118, 0.9900616407394409, 0.1254691779613495, 1.8848778009414673, 1.5547112226486206, -1.255525827407837, 0.5592740774154663, 1.0985432863235474, -0.5519374012947083, 1.927301287651062, 0.23375092446804047, -0.6907164454460144, -1.4972203969955444, 0.2019040435552597, 0.6630934476852417, 0.4546810984611511, -1.1385257244110107, 0.7681317329406738, -0.821995735168457, 1.2658864259719849, -1.86680269241333, 0.4483049213886261, 0.5659316778182983, 0.22532062232494354, 2.424745559692383, 0.8151351809501648, 0.13790589570999146, -0.7486706376075745, 0.19381172955036163, -0.9298839569091797, 0.6476069092750549, -0.5009725093841553, -0.05904872342944145, 0.5015192031860352, 0.4051750600337982, 0.8076919317245483, 2.027869701385498, 0.22727735340595245, 0.3616213798522949, -1.6079522371292114, 1.0944607257843018, -0.914317786693573, -0.8547298312187195, -0.7036650776863098, -0.6217673420906067, 0.48658305406570435, 2.195828914642334, -0.7073959708213806, 1.0551823377609253, 0.8274797201156616, -0.13182656466960907, -0.584648847579956, -0.047303956001996994, 0.25331300497055054, -2.284669876098633, -0.5975802540779114, -0.02100975252687931, 0.3891306519508362, -0.36126264929771423, 0.43968087434768677, -1.1014372110366821, 1.2642815113067627, -0.5228524804115295, -0.5872984528541565, -0.9355812072753906, -1.6516140699386597, 0.37559619545936584, -1.5348284244537354, 0.09131360799074173, 0.44634464383125305, -0.03230088949203491, -2.8142263889312744, -1.7792599201202393, 0.14328807592391968, 0.4856480658054352, -0.8702080845832825, 0.0969742089509964, 2.0127153396606445, 1.0834736824035645, -1.5747982263565063, -0.06172376126050949, 0.6653196215629578, -0.7482801079750061, -0.8796515464782715, -1.1518357992172241, -1.247149109840393, 1.0903255939483643, 1.090563178062439, -0.46191370487213135, -2.420063018798828, 1.0325026512145996, -0.826272189617157, -2.4948313236236572, -0.5573644638061523, -0.6877370476722717, -0.3214406967163086, 1.202699899673462, 1.190563678741455, 1.6122008562088013, -1.2133028507232666, 1.4340789318084717, 0.5602330565452576, 1.4970152378082275, -0.6186243891716003, -1.0846706628799438, 0.36227092146873474, 0.9871365427970886, 0.06285945326089859, 0.5549881458282471, 1.6988784074783325, 0.4387998878955841, -0.16451948881149292, 0.22121378779411316, 0.7940075397491455, -0.6095663905143738, -1.7433626651763916, -1.4641505479812622, -1.6012500524520874, -0.8229994177818298, 1.3312556743621826, 0.32775822281837463, -0.627213716506958, 1.3996604681015015, 1.4174377918243408, 0.7985230088233948, 0.7585469484329224, -0.6241202354431152, -0.7284528613090515, 2.026723623275757, 1.2440296411514282, -2.2176332473754883, -0.3665368854999542, 0.447115957736969, 1.3236403465270996, 2.8076255321502686, -0.2935663163661957, 0.7216894626617432, -0.7857767939567566, -0.15459482371807098, 0.4680176079273224, 0.4284387528896332, 0.943846583366394, 0.11379978805780411, 0.37100622057914734, 1.4261771440505981, 0.7246943712234497, -1.0490989685058594, -0.05907903611660004, 0.09359321743249893, 1.964367389678955, 1.5282214879989624, -1.8068212270736694, 0.5180922150611877, 1.1618845462799072, -2.311418294906616, -0.6187069416046143, 2.374704599380493, -0.3497564494609833, -1.6318960189819336, 0.6916817426681519, -0.8678382635116577, -0.33358585834503174, 0.854170024394989, -0.842118501663208, -2.1184678077697754, -0.4529382884502411, 1.120957374572754, -0.6103838086128235, -0.8214579820632935, -0.47858330607414246, 0.8603466153144836, 0.8660275936126709, -0.36522725224494934, -0.3801827132701874, 0.12139509618282318, -0.3354569375514984, -0.763390839099884, -0.9097173810005188, 1.6493629217147827, -0.21369220316410065, 0.14973926544189453, -0.23499219119548798, -3.348625659942627, 1.0049808025360107, -1.493152379989624, -0.36402928829193115, -0.6778966188430786, -1.6198145151138306, 1.2424026727676392, -0.9665929675102234, 0.39072665572166443, -0.28642818331718445, -2.2126283645629883, -0.08035137504339218, 0.1543145477771759, 1.0505714416503906, -1.2426767349243164, 0.46510428190231323, 0.47794443368911743, -2.4674484729766846, 0.2571128010749817, -1.9225929975509644, -0.2011343091726303, -0.4112042188644409, -0.2802172303199768, -1.0692251920700073, -0.9386151432991028, -0.7880921959877014, 0.6559567451477051, -0.4272625148296356, 0.2612505555152893, 0.1669279932975769, -0.7757468223571777, -0.6869942545890808, -0.8967210054397583, -0.1364765465259552, 1.2949697971343994, -2.369187116622925, -1.0177292823791504, 0.8317016959190369, 1.1710602045059204, -0.1630495935678482, -0.6888315677642822, 0.050153546035289764, 1.0545657873153687, -0.9924626350402832, -0.410193532705307, -1.5568941831588745, -1.153835654258728, -0.8642945885658264, 0.4855576753616333, -0.5219572186470032, 0.04242907091975212, 1.1782430410385132, 0.05518752336502075, 1.3141303062438965, 0.4352571368217468, -0.1306212991476059, -1.3309054374694824, 0.5830892324447632, -0.08678526431322098, 0.7207247614860535, 0.11695663630962372, -0.26101115345954895, -1.5819131135940552, 0.1833319365978241, -1.7727744579315186, 0.4870181977748871, -1.099120855331421, -0.17328709363937378, -0.11120565235614777, -0.7807499170303345, 0.8158148527145386, -0.5096392035484314, 0.24066728353500366, 0.2046050876379013, 0.5063937902450562, -0.4523473083972931, -0.41914018988609314, 1.9964103698730469, -0.2713722586631775, -0.1387365162372589, -0.6496731042861938, 1.4536510705947876, 0.8977721929550171, 0.6939339637756348, 0.4792110323905945, 0.7068935632705688, -0.5085436701774597, 0.8462314605712891, 0.015717346221208572, -0.6610494256019592, 0.5900622010231018, 0.9196267127990723, 0.29237663745880127, 0.2371530383825302, -0.8668508529663086, 0.0834919810295105, 0.5150303840637207, -0.2411259561777115, -1.456685185432434, 0.1904188096523285, -0.7327225804328918, -1.8051871061325073, 0.2113126963376999, -1.377305269241333, -2.2364614009857178, -0.7442880272865295, -1.6521830558776855, -0.41059714555740356, -0.07997997105121613, -1.53959059715271, -1.043868064880371, 1.381840705871582, -0.6564267873764038, 0.25520965456962585, 0.5913865566253662, 0.2525489330291748, 0.23375415802001953, 2.1651811599731445, 0.6268592476844788, -1.3801372051239014, 1.4941462278366089, 0.22626183927059174, -0.10183937102556229, 1.3955583572387695, 1.1790965795516968, 1.3386859893798828, -0.6013332009315491, 1.0184298753738403, -0.398269921541214, 0.8352334499359131, 1.076608419418335, -0.4338136315345764, -0.14721088111400604, 1.1089396476745605, 0.46611347794532776, 1.1130032539367676, 0.5841984748840332, 0.34247803688049316, 0.7867099046707153, -0.49007102847099304, 0.5492751598358154, 1.8631490468978882, -0.15126903355121613, 0.3067638874053955, 0.15341654419898987, -1.2653679847717285, -0.08201440423727036, -0.28100720047950745, -1.880140781402588, -0.9739070534706116, 1.2469202280044556, 1.294394612312317, -0.7981520295143127, -1.1561102867126465, -0.28330397605895996, 0.6856136322021484, 1.3597179651260376, 0.6082480549812317, -2.4508330821990967, -0.6997443437576294, -0.17240607738494873, -0.4130595326423645, 0.08179157227277756, 0.19921797513961792, -1.0670356750488281, 0.3199537992477417, 0.31441813707351685, 0.09629553556442261, 0.811739981174469, -0.9388697147369385, 1.2304272651672363, -1.5827419757843018, 0.6134806871414185, 1.1227473020553589, 0.4856218993663788, 0.28128525614738464, -1.1654422283172607, -0.17659664154052734, -1.044214129447937, -2.443439245223999, 1.2348750829696655, 1.035507321357727, -0.848206102848053, 1.772278070449829, 2.587367057800293, -1.161523699760437, 1.9566420316696167, -0.5834885835647583, -0.39926448464393616, -0.8759427666664124, -0.7727715373039246, -1.5829219818115234, 1.4681284427642822, -0.03772304579615593, 0.417174756526947, 0.4716106355190277, -0.012422513216733932, -0.23179054260253906, 0.4843846261501312, -1.0419129133224487, -0.18453748524188995, -1.9504055976867676, -1.3469127416610718, -0.22595429420471191, -0.5043550729751587, -0.45250093936920166, 0.6797317862510681, 1.2602468729019165, -0.5155333280563354, -1.0164297819137573, -1.874696969985962, -1.264601230621338, 1.3885551691055298, 1.036004900932312, -0.5599617958068848, 2.0224199295043945, 1.6324690580368042, 1.432631492614746, -0.31534621119499207, 0.5937479734420776, -0.36702442169189453, -1.158759355545044, -0.22988171875476837, -0.8432010412216187, 1.2182685136795044, 0.13420796394348145, 0.8944928050041199, 0.39561739563941956, 0.5985905528068542, -0.845683217048645, 0.4135497808456421, 1.8620625734329224, -1.096487283706665, 0.17066258192062378, -1.7855486869812012, 0.23943056166172028, -0.059860046952962875, 1.642839789390564, -0.2862240970134735, 0.4207872152328491, 0.25063392519950867, -0.9679741263389587, 1.622496485710144, -0.15875452756881714, -0.12348473817110062, 0.24377770721912384, -0.17637430131435394, 2.282226800918579, 0.49618905782699585, -0.14526230096817017, -0.37424585223197937, -0.5690940022468567, -0.30256691575050354, 0.7161288857460022, -0.9200485944747925, -1.267112374305725, -0.418897420167923, -1.045853853225708, 0.9602891802787781, 0.8309505581855774, 1.3667904138565063, 0.017765352502465248, 0.5542321801185608, -0.04961415380239487, -1.0687559843063354, -1.4290186166763306, -0.8735207319259644, 0.623833417892456, 0.5533697605133057, -0.475960910320282, 0.4858328402042389, 0.5442271828651428, 0.9069328904151917, 0.34540411829948425, 1.457400918006897, 1.673629641532898, 0.3300865888595581, 0.26223549246788025, -1.1528520584106445, 0.5886857509613037, -0.46878695487976074, 0.9065341949462891, 0.8436617255210876, -0.3281671404838562, 0.7634742856025696, -0.3925494849681854, -0.5422270894050598, -0.548773467540741, -1.5057485103607178, 0.811237096786499, 1.3298661708831787, -1.3611735105514526, -1.0733730792999268, 0.8009815216064453, 0.4087734520435333, 0.5586944222450256, 0.4829782545566559, 0.26884573698043823, -0.11267688870429993, -0.6000850796699524, -0.9908996224403381, 1.0296518802642822, -0.6390799283981323, -1.382325530052185, -1.2714399099349976, 0.0442279689013958, 0.28928688168525696, -0.34598204493522644, 1.0338174104690552, 1.0522750616073608, 0.2824319005012512, 0.11356669664382935, -0.217612624168396, -0.8359585404396057, 1.0431621074676514, 0.37915903329849243, -0.5245426297187805, -0.3231687843799591, -0.4797840714454651, 0.44571608304977417, -0.4861017167568207, 0.14592193067073822, -0.3615931272506714, -0.5364070534706116, -0.76934814453125, 0.27398058772087097, 0.9085451364517212, 0.05232550576329231, -0.976114809513092, 0.3955085277557373, -0.28789061307907104, 0.39754679799079895, 0.49030566215515137, -0.7388142943382263, -0.732134997844696, 0.55695641040802, 0.031312935054302216, 1.275278091430664, 0.8275641202926636, 0.14405536651611328, 1.176315426826477, -0.5398439764976501, 0.5409754514694214, 0.8766154050827026, 0.6238489747047424, -0.2814021110534668, 1.5008097887039185, 0.29572567343711853, -0.4881211519241333, 0.6865149736404419, 0.1941567212343216, -0.23907315731048584, 0.3457343578338623, -0.32140907645225525, 0.7699283957481384, -1.4445374011993408, -0.416951060295105, 0.965918242931366, 0.47578567266464233, -0.08910731226205826, 1.3784916400909424, 1.3147284984588623, 0.7547799944877625, -0.4204873740673065, 0.342864066362381, -1.3424427509307861, 0.007335660047829151, -0.7186203598976135, -0.4133962094783783, -0.10683261603116989, -0.11639664322137833, -0.6654095649719238, 0.32549819350242615, 1.1945067644119263, 0.6289065480232239, 0.8277329206466675, -0.9516479969024658, -0.8039917349815369, -1.038008689880371, -0.6772353649139404, 0.8839827179908752, 1.0344264507293701, -1.4215478897094727, 1.0739396810531616, 0.9145560264587402, -0.2721981406211853, 0.7608652710914612, -0.9957371354103088, 0.02635195665061474, 1.3143446445465088, -1.580108404159546, -0.90260249376297, 0.6303203105926514, 0.4127349853515625, -0.06155398488044739, 0.9484284520149231, 1.1164799928665161, 0.6575829982757568, -0.3654920160770416, 0.15378767251968384, 1.831982135772705, -0.5061453580856323, -0.09454032778739929, 1.3495041131973267, -0.541390597820282, -0.47553738951683044, -0.007182378321886063, 0.5508652925491333, -0.3579773008823395, -0.1556011587381363, 1.3970247507095337, -0.588843047618866, -0.20828641951084137, 0.6177629232406616, 0.46124371886253357, 0.9956119060516357, 0.6199504137039185, 0.12883000075817108, -1.2388285398483276, -0.11376754194498062, 0.22671285271644592, -1.1085783243179321, 0.7921519875526428, 0.7300821542739868, -0.8651849627494812, 0.8269136548042297, 1.1123923063278198, -0.5048638582229614, -0.24421252310276031, 0.892135500907898, -0.8608828186988831, -0.19594626128673553, -0.5056895613670349, 0.2305150181055069, 1.6104931831359863, 0.3734835982322693, 1.8125410079956055, -0.27180299162864685, -0.38678863644599915, -0.5081983804702759, 0.09035778790712357, -0.29504281282424927, 0.14186759293079376, -0.8084520697593689, -0.8072948455810547, -1.392460823059082, 0.29909902811050415, -1.0847302675247192, -0.8407954573631287, -0.8841130137443542, 0.2250998467206955, 1.132348656654358, 0.45092839002609253, -0.7703917026519775, -0.6013548374176025, 0.23513725399971008, -0.031138457357883453, 1.4448111057281494, -1.6112277507781982, 0.6678726673126221, -1.1059074401855469, -1.3733949661254883, -0.19793081283569336, -0.705421507358551, -1.971941590309143, -1.122471809387207, 0.9160621762275696, 0.2424914538860321, -1.4777745008468628, -0.613075315952301, 0.49351635575294495, 0.6463493704795837, 0.9380630254745483, 0.7770420908927917, -0.2692236602306366, 1.2324391603469849, -0.8472623229026794, 1.1402015686035156, -1.863857626914978, -1.4682940244674683, 0.6825253367424011, 0.2822280526161194, 0.03249999135732651, -1.1930862665176392, -0.044179365038871765, -1.7798153162002563, 1.1826337575912476, 1.1994227170944214, -0.628402590751648, -0.5636504292488098, -0.43352800607681274, 0.8414876461029053, 1.241890788078308, -1.6562937498092651, 0.37057000398635864, 0.331246942281723, 0.6616179347038269, -0.913465142250061, -0.226554274559021, -0.7672077417373657, -0.5661970376968384, 0.15166907012462616, -0.1681259125471115, 0.2939014434814453, -0.20365840196609497, 2.242199182510376, 0.21262869238853455, 1.1352262496948242, 1.7998958826065063, 0.48296549916267395, 0.25037798285484314, 0.46795353293418884, 0.9932575821876526, 0.1440693587064743, -1.0014318227767944, 1.322105884552002, 0.5049431324005127, 0.1439744532108307, 0.07648205757141113, 0.7607711553573608, -0.19598804414272308, 0.5545448660850525, -0.08550399541854858, 0.5383392572402954, -0.9206985831260681, -0.30542102456092834, -0.8953337669372559, -0.2379586398601532, -0.4655259847640991, -0.3698003590106964, 0.5385591387748718, 0.21452754735946655, -0.9790452718734741, -0.07534454017877579, -0.7488137483596802, 0.20821954309940338, 0.05564353987574577, 0.2562825679779053, -1.1802513599395752, -0.2953771650791168, -0.2380850613117218, 1.6194884777069092, 1.1404730081558228, -0.8320808410644531, 1.2406063079833984, 1.9549376964569092, -0.5434406399726868, -0.4801292419433594, -0.015114721842110157, -0.20716309547424316, -0.1744430959224701, -1.3574398756027222, -0.42942380905151367, -0.42710086703300476, 0.441262423992157, -1.5736162662506104, 0.9838690161705017, 0.8952786922454834, -1.8425593376159668, 0.27471691370010376, 0.0033793800976127386, 1.1883679628372192, -1.7503410577774048], [0.6808498501777649, 0.7946094870567322, 2.231051206588745, 1.1757116317749023, 0.42178088426589966, 0.2673882246017456, 0.14977216720581055, -0.18982988595962524, -0.33044567704200745, 1.9149067401885986, 0.8219946026802063, -0.48588937520980835, 0.0213939119130373, -0.923402726650238, 0.6931229829788208, -0.2840883731842041, -0.42734137177467346, -0.7031097412109375, 0.45935824513435364, -2.088559150695801, 0.5298520922660828, 0.6578539609909058, -0.19977810978889465, -1.0985636711120605, -0.0666470155119896, 0.8319057822227478, 0.9664057493209839, 0.3501434326171875, -1.2786390781402588, -1.668199896812439, 1.4788542985916138, 0.8443312048912048, 0.3344212472438812, -0.7085840702056885, -0.028230955824255943, -0.22413448989391327, -0.03806212544441223, 0.4265497922897339, 0.4227331280708313, 0.10621237754821777, 0.9011037945747375, 1.2314035892486572, 0.2298951894044876, -2.685760974884033, -0.6505815386772156, 0.16892117261886597, 0.7222751379013062, 0.025221850723028183, -0.09169939905405045, 0.03979859501123428, 1.5323665142059326, 1.384130597114563, 1.8236156702041626, -0.5248770117759705, -0.9013241529464722, 0.05962473154067993, 0.21205146610736847, 0.43746039271354675, -0.7397581934928894, -0.11449427902698517, 0.17798689007759094, 0.9129872918128967, 0.446312814950943, -0.38031044602394104, -0.32916536927223206, 0.26853087544441223, -2.7222654819488525, -0.9265075325965881, 1.5487918853759766, -0.19859816133975983, -1.2468640804290771, 0.7457671761512756, -0.9291515946388245, 0.08597618341445923, -1.504021406173706, 0.7495100498199463, 0.49546512961387634, -0.7890264391899109, -0.21292328834533691, -0.36978113651275635, -1.7451502084732056, -0.38085296750068665, -0.2474655956029892, 1.8153502941131592, 0.23713697493076324, 1.137377142906189, -1.371466040611267, -1.3687800168991089, -1.5586665868759155, -1.1366620063781738, 0.6270185112953186, -0.35091009736061096, 0.7955706119537354, -1.1389065980911255, 0.24424001574516296, -0.3422391414642334, -2.5856330394744873, -0.9831516146659851, -0.20127369463443756, 0.10663016885519028, -0.6697059273719788, -1.14505136013031, 0.6216583847999573, 1.3221627473831177, 0.7811633944511414, 0.8693253993988037, 0.9825872182846069, 0.6804636120796204, -1.0356849431991577, 0.508293092250824, 0.716096043586731, 0.4679734408855438, -0.6069024801254272, 0.34291526675224304, -2.1535208225250244, -0.6070064902305603, -0.7948709726333618, -0.6414464712142944, 0.09821223467588425, 1.0124742984771729, -0.9223293662071228, 0.8971518874168396, 1.1829750537872314, -0.6426523327827454, 1.0369266271591187, 1.5023736953735352, -0.2444027066230774, -0.5205246210098267, 1.5931748151779175, 1.7423111200332642, -0.08084601908922195, -0.03731868788599968, 0.32264846563339233, -0.2327338010072708, 0.4437974691390991, -1.7192740440368652, 0.055346064269542694, -1.2416404485702515, -0.5017214417457581, -0.1886998862028122, 0.6753302812576294, -0.2915535867214203, -0.465593546628952, -0.024217812344431877, 0.40428122878074646, -0.2865886986255646, -1.0388644933700562, 1.2919423580169678, -1.181589961051941, 0.03854416683316231, 0.8056694269180298, -0.8504595756530762, -1.8870439529418945, -0.1261601597070694, -1.0286273956298828, 1.3458229303359985, -1.4357320070266724, 0.10323058068752289, -0.38774988055229187, 0.4985960125923157, -0.46126314997673035, 1.0038321018218994, 1.2775111198425293, 0.941926896572113, -0.675620436668396, 0.5041642189025879, 1.5640977621078491, -1.5949053764343262, -0.6201040744781494, 0.5172836184501648, -0.3775020241737366, -0.42967018485069275, 1.772078275680542, 1.2868835926055908, -0.7952818274497986, 0.936954140663147, -1.1724227666854858, 1.085803747177124, 0.10812384635210037, -0.9241650700569153, -1.0539833307266235, 1.0893690586090088, -2.0113775730133057, 0.14754416048526764, 0.5425041317939758, 1.3263620138168335, 0.8797469139099121, 0.6247439384460449, 1.2782186269760132, 2.284179210662842, 2.7240312099456787, 0.5800483822822571, -1.0135473012924194, -0.10036872327327728, 1.4917296171188354, 0.9852791428565979, 0.7836591005325317, 1.8485865592956543, 1.7826082706451416, -0.8583642244338989, -0.6635865569114685, 0.41871577501296997, 0.11234378814697266, 0.949416995048523, -1.5266331434249878, 1.3817580938339233, 1.0000722408294678, 1.8864377737045288, 0.7521548867225647, -0.2439160943031311, -0.17083340883255005, 0.6597320437431335, -0.5169855952262878, -0.15882094204425812, -0.6619607210159302, -1.3079991340637207, -0.10488554835319519, -0.07866667211055756, -0.32482582330703735, -0.04141975939273834, -0.7514175176620483, 2.455448865890503, 0.7418053150177002, -0.9457770586013794, 1.2365107536315918, -0.4920823276042938, -0.9429569840431213, 0.6714156270027161, 1.092499017715454, -0.298005074262619, 0.5086779594421387, -0.12678228318691254, 0.5223784446716309, -1.4255433082580566, 0.40422946214675903, -1.3535405397415161, -0.003135370323434472, -0.7952955365180969, -0.3681805729866028, -0.5349234938621521, 0.7421917915344238, -0.3797264099121094, 0.9527802467346191, 0.23381897807121277, 1.0103400945663452, -0.2750795781612396, 0.5990501046180725, -0.8249258399009705, -1.3199217319488525, -1.1837538480758667, 0.1426016390323639, -0.3301309645175934, 0.32769325375556946, -1.302038550376892, 0.5947794914245605, -0.9863901734352112, -0.4448925852775574, -0.9725040793418884, 1.0453778505325317, -0.4836317300796509, -0.649760365486145, 0.5846807360649109, -2.2735064029693604, 1.4128795862197876, -0.5391831398010254, -1.108738899230957, 0.6809582114219666, 0.120451420545578, -0.8083174824714661, -0.27752432227134705, -0.3078382611274719, -0.6388309001922607, -0.998022735118866, 0.8449118733406067, 2.1892800331115723, -0.17263099551200867, 0.6132734417915344, 1.0879950523376465, 0.8729772567749023, 0.38842374086380005, 1.4538925886154175, 2.9518680572509766, 0.458590030670166, 0.004362551961094141, 0.18086855113506317, -0.08213333785533905, -0.7106845378875732, -1.248534083366394, -0.04993949830532074, -1.1368625164031982, 1.6139593124389648, 0.08809363096952438, -0.9885839223861694, -1.393924593925476, -0.48189517855644226, -0.46454721689224243, 0.7706356644630432, -0.8085146546363831, -0.5993916988372803, 0.2969817817211151, -0.16092301905155182, 0.20472452044487, -0.09927614778280258, 1.2059509754180908, -1.6482963562011719, 0.8223807215690613, 1.177517056465149, 1.1773556470870972, -0.781318187713623, -1.4325100183486938, 0.1567697674036026, 1.0155893564224243, 0.26929768919944763, -1.5563454627990723, 0.4669361114501953, 1.5792399644851685, -1.7758697271347046, 1.4931107759475708, 1.3220309019088745, 1.6463514566421509, -0.36873045563697815, 0.8749322295188904, 2.0011355876922607, -0.36661669611930847, 2.054126262664795, 1.0739120244979858, 0.3858175277709961, -0.5991852283477783, 0.7045392394065857, 2.290043592453003, 1.4519829750061035, -1.0556758642196655, -0.6605038642883301, 0.6108596920967102, 0.9518558979034424, 1.6587202548980713, 0.7880008220672607, -0.561917781829834, -1.5960737466812134, -0.8838745355606079, 0.8907337784767151, -0.48683223128318787, 0.41983920335769653, 0.16569896042346954, 0.1113390326499939, 0.04441484436392784, -0.4346420168876648, -0.9688582420349121, 1.3703378438949585, -1.6896060705184937, 0.9250918626785278, 0.30194398760795593, -0.358650267124176, -1.4686847925186157, -1.4142955541610718, -0.8273540735244751, 1.042716145515442, 0.010600321926176548, -1.8442760705947876, 0.3325716555118561, -1.0288172960281372, -0.37716203927993774, 0.7075127363204956, -0.7785763740539551, 0.9802175760269165, 0.9452569484710693, -1.8181246519088745, 0.08079652488231659, 0.3646351099014282, -0.7411036491394043, -0.9680337309837341, -0.8270675539970398, 1.4841517210006714, -2.8014473915100098, -0.5850364565849304, -0.9177431464195251, -0.0735224261879921, -1.0656201839447021, 0.9771667122840881, -0.7710753083229065, -1.1928737163543701, -0.5694381594657898, 0.6138722896575928, 0.40365666151046753, 0.31857988238334656, -1.0646363496780396, -0.25037339329719543, 0.14222301542758942, 0.15932407975196838, -0.39718344807624817, 1.0632777214050293, 1.8013371229171753, -0.39826786518096924, 1.3814729452133179, -0.6009775996208191, -0.7581927180290222, -0.9566438794136047, 0.6292877197265625, 1.2215502262115479, -0.37246251106262207, 0.5308617949485779, 1.4482694864273071, 0.4036664068698883, 0.4342523217201233, -0.9998863339424133, -0.34520694613456726, 0.6760307550430298, 0.4625181257724762, 0.4958043694496155, -1.89298677444458, -1.6515804529190063, -0.21041515469551086, -0.21820887923240662, 1.093729853630066, -1.0445483922958374, 1.857870101928711, 0.06458883732557297, -0.23799379169940948, 0.8864917755126953, 0.7137749195098877, -0.8423351049423218, 0.37512001395225525, -1.1275001764297485, -1.6056709289550781, -0.2119603008031845, -1.025889277458191, 0.2662334144115448, -0.13263924419879913, 0.21052923798561096, -1.145461082458496, -1.1302268505096436, 0.42277905344963074, 2.197697639465332, -0.7119767069816589, -2.5422377586364746, -0.27621012926101685, -0.034440137445926666, 1.5196572542190552, -0.4590381681919098, 1.4715821743011475, -0.7312315106391907, -0.05955236032605171, 0.7502520084381104, -1.7464607954025269, -0.2656109929084778, 0.748577356338501, 1.1722038984298706, -0.6183083057403564, -0.34095528721809387, -0.3371903598308563, 0.713525652885437, -0.09500882774591446, -2.6821706295013428, 0.0980134904384613, 0.202698215842247, -0.3135594129562378, 1.0641635656356812, -0.4103003740310669, 0.25398242473602295, -0.4157213568687439, 0.4344511032104492, 0.6144016981124878, -0.1670066863298416, 0.008056514896452427, -0.3826466500759125, 0.8579146265983582, 1.6676785945892334, -0.7185689210891724, -1.2908316850662231, -1.069981575012207, -0.095547616481781, -0.6036468148231506, -2.200971841812134, 1.042210578918457, 0.9669821262359619, 0.3246728181838989, -0.8275932669639587, 0.689044177532196, -0.25049588084220886, 0.7735117077827454, -0.46557921171188354, -0.04913201183080673, 1.3359318971633911, -0.46470025181770325, 1.1035308837890625, 0.7814367413520813, 1.4638391733169556, 0.17672280967235565, 1.162672519683838, 0.3795062303543091, -0.14457999169826508, -0.6484476327896118, -1.3802387714385986, -0.5798946022987366, 3.0936405658721924, 0.5022121071815491, 0.6689295768737793, -2.4105331897735596, -1.558681607246399, 0.35719937086105347, -0.958118736743927, 0.9331841468811035, 2.1107192039489746, 1.188068151473999, -0.9167788028717041, 0.5127118825912476, -1.2252397537231445, 0.587346076965332, 1.8928278684616089, -0.161544069647789, 0.05118690803647041, -0.4314914345741272, 0.4607819616794586, 1.5210721492767334, 0.407942533493042, -0.3351723551750183, -1.270053505897522, 0.22006163001060486, 0.8339235186576843, 2.4519896507263184, 0.7440184950828552, 0.9841703176498413, 1.4702184200286865, 2.8315494060516357, -0.11580755561590195, -0.34156593680381775, 0.6508952379226685, -2.125239372253418, -0.9188293814659119, 1.71104896068573, -0.10691609233617783, -1.3638454675674438, -0.1106468215584755, -0.48686113953590393, -0.9809141159057617, 0.04658733680844307, 1.1679527759552002, -1.34145188331604, -0.3989451229572296, 1.45673668384552, -1.5277541875839233, 1.917051911354065, 0.16054131090641022, 0.6648238301277161, -0.26199427247047424, 0.524285078048706, -1.9722754955291748, -0.6032471060752869, 0.20476332306861877, -1.6153393983840942, 0.7532374262809753, -1.9536978006362915, 0.596053421497345, 0.4921460449695587, 0.6798034906387329, -0.638670802116394, 1.781135082244873, -1.3388162851333618, -1.2407399415969849, -0.9660712480545044, 0.8630096316337585, 0.9026546478271484, 0.6879517436027527, 0.33316391706466675, -1.6181477308273315, 0.30505266785621643, 0.9494447112083435, -0.057080481201410294, -0.9239468574523926, 1.4070446491241455, 0.3154006600379944, -1.7436131238937378, 0.5054385662078857, -0.181244358420372, 1.8693134784698486, -0.9122510552406311, 0.1261550337076187, 0.18617984652519226, 1.507691502571106, -1.5684257745742798, 1.7114747762680054, 2.5643932819366455, -1.0995904207229614, 0.18092778325080872, 0.8528157472610474, 0.7799839973449707, 0.01704251579940319, -0.26521363854408264, -0.0370260514318943, 0.6226654052734375, 1.227394700050354, -1.1988779306411743, -0.5467576384544373, -0.9249125123023987, 0.7580908536911011, -0.9225266575813293, -0.22220464050769806, 0.9158118367195129, 0.46537673473358154, -0.9241721630096436, -1.0581611394882202, 1.293054461479187, -0.4497142434120178, -1.197751760482788, 1.9954544305801392, 0.2197311669588089, -0.29355719685554504, -1.1019175052642822, -0.21833381056785583, -1.4111928939819336, 0.26614511013031006, -0.9092841148376465, -0.06939010322093964, 0.7396062612533569, 0.12048172950744629, 1.9025770425796509, 0.9350537657737732, 1.317726731300354, -0.2884054183959961, 1.3453075885772705, -0.20951701700687408, -1.0958232879638672, -0.4237699508666992, 0.18635448813438416, -1.188101053237915, 0.5677376389503479, -0.706063449382782, -1.6215354204177856, -0.41982501745224, 1.2071927785873413, -0.4238726794719696, 0.5885522961616516, 0.46183648705482483, 0.13315735757350922, 1.6278475522994995, 1.522274374961853, -0.3061635196208954, -1.0627408027648926, -0.7517630457878113, -1.9891237020492554, -0.012034235522150993, 0.6010459065437317, -1.917413353919983, -0.8071284890174866, -0.8269291520118713, 1.0713196992874146, -0.4572308659553528, -0.8782504200935364, -0.4004683792591095, 0.8695447444915771, -0.7189382314682007, 1.3959465026855469, -0.9623751640319824, -0.8783213496208191, -0.4969743490219116, -0.08233196288347244, -0.6510944366455078, 0.07412846386432648, 1.9337965250015259, 0.13814836740493774, -0.030120328068733215, 1.4905966520309448, 0.5580435395240784, -0.35626456141471863, 2.037806749343872, -2.302773952484131, 1.4587690830230713, -0.32370707392692566, 0.07046680152416229, 1.233506441116333, -0.39777204394340515, 0.5471377372741699, 0.3301776349544525, -0.9571211934089661, 1.593807339668274, -0.2985419034957886, 0.09660081565380096, 0.0032715986017137766, 0.16761058568954468, 0.3549947738647461, 1.354448914527893, -1.37814462184906, 0.7081746459007263, 2.452491521835327, -1.3924683332443237, 0.8803200125694275, 0.6808738112449646, -1.5409520864486694, -0.7473832368850708, 0.4047715663909912, 1.2240194082260132, -0.9591320753097534, -1.2414358854293823, -0.04765676334500313, -1.4796180725097656, 0.5921591520309448, -0.6307026743888855, 0.582153856754303, 0.28481459617614746, 0.7303723096847534, 1.4346208572387695, 1.2728652954101562, -0.07488586008548737, -1.4041690826416016, 0.21869033575057983, -2.654266119003296, 1.218877911567688, 0.5700619220733643, 1.8484692573547363, 0.42840370535850525, -1.5471739768981934, 1.4540996551513672, 0.7375263571739197, -1.3370885848999023, -0.25502079725265503, -0.6641080379486084, 0.0657094269990921, 0.6841172575950623, 0.32939890027046204, -1.1013065576553345, -0.18608757853507996, 1.1702529191970825, 0.3259117901325226, 0.07244384288787842, -0.5896807312965393, -0.6539013981819153, 0.2562282681465149, -1.635297179222107, 0.5263324975967407, 0.41544395685195923, -1.5051114559173584, 1.247058629989624, -0.9051343202590942, 0.4398673176765442, -1.3398125171661377, -1.2808107137680054, -0.31375542283058167, 1.1673035621643066, -0.873593270778656, -0.8981360197067261, -0.3753761947154999, -1.091947317123413, -0.5100973844528198, -1.48292076587677, 0.4840553402900696, -0.4093846380710602, 0.31255239248275757, 1.1772964000701904, 0.2528240978717804, 0.0934402197599411, -1.1642212867736816, -0.08784060180187225, 1.759992241859436, -1.031559944152832, 0.4386342763900757, -0.39048832654953003, -0.4832727015018463, -0.8454345464706421, -0.5745728015899658, -1.266486406326294, 1.495398759841919, -0.5959452390670776, 0.618958055973053, 0.42081207036972046, 0.08118728548288345, 0.09786835312843323, 0.052786849439144135, -0.45954564213752747, 0.9273896217346191, -0.06230112537741661, 0.8287030458450317, 0.349650114774704, 0.49411699175834656, 0.7899439930915833, -0.609359622001648, 0.35235369205474854, 0.294341117143631, -0.3330063819885254, 0.29033079743385315, 0.033227916806936264, 0.6374353766441345, 2.7350316047668457, 0.7219983339309692, 0.18582992255687714, -1.166229486465454, 1.1142877340316772, 0.25586092472076416, 0.38728684186935425, 0.436714768409729, -0.06713556498289108, -0.6198284029960632, -1.0285929441452026, -0.5335267782211304, 0.4241783916950226, -0.03920315206050873, 0.8393246531486511, 1.4532365798950195, 0.04292610287666321, -0.5892651081085205, 1.3607820272445679, 0.6561436653137207, -0.009203094057738781, 0.7151582837104797, -0.09363584220409393, 0.6768496036529541, 0.6847324371337891, 0.13765238225460052, -0.9214670658111572, 0.9987305998802185, -0.35283803939819336, -0.565920352935791, -0.1247534528374672, -1.2903614044189453, -0.2657187283039093, 0.45011040568351746, 0.6298460364341736, 1.0963377952575684, 1.4173917770385742, 0.2757967412471771, 1.6455870866775513, 0.20687983930110931, -0.3129119873046875, 1.5624841451644897, 1.2558026313781738, -0.4516236484050751, 0.313424676656723, -0.16069935262203217, 0.7450289130210876, 1.3450149297714233, 0.07224322110414505, -1.0229023694992065, -0.007376053836196661, -1.1194484233856201, -0.26377159357070923, -1.4633115530014038, 0.5236172080039978, -0.5195714235305786, -0.8395851850509644, 1.098062515258789, -0.17986257374286652, 0.9097389578819275, 0.6112134456634521, 0.5068084597587585, 0.20737537741661072, 2.1117570400238037, 0.6123664379119873, -0.78159099817276, -0.1830264925956726, -0.03704357519745827, 0.013254855759441853, 0.6756773591041565, 1.101976990699768, -0.9439799189567566, 0.3694068491458893, -0.7789354920387268, 0.1768747717142105, 2.0380029678344727, 0.13051435351371765, 0.6006473898887634, 0.30730560421943665, -0.3039272725582123, -0.03689727932214737, -0.1949528306722641, -0.36509445309638977, -0.7431014180183411, 0.26821523904800415, -0.8815375566482544, -0.34350693225860596, -2.354996919631958, -0.17508749663829803, 0.07461026310920715, -2.081289052963257, 0.7824317216873169, 2.3595433235168457, 0.7688027024269104, -2.6573476791381836, -2.0013480186462402, 1.0032804012298584, -0.7488631010055542, 0.4524497389793396, 0.6945039629936218, 0.33070969581604004, 0.6927611231803894, -0.01234844047576189, 0.17125247418880463, -0.9124001264572144, -0.45152634382247925, 0.2809157371520996, -1.463689923286438, -0.20872238278388977, 2.070573329925537, 0.020035535097122192, -0.10167977958917618, -1.6669443845748901, -1.1518474817276, -0.7353457808494568, -1.3678011894226074, -1.5002973079681396, -1.998227596282959, 1.1423290967941284, 0.8431674838066101, -1.1218832731246948, 1.0198715925216675, 0.9102197885513306, -1.2386424541473389, 0.9147112965583801, -0.8927313685417175, -0.06967008858919144, -0.15810167789459229, -1.3636242151260376, 1.2499538660049438, 0.978080153465271, 1.4063640832901, -0.5075036883354187, -0.6547663807868958, -0.1287325918674469, 0.7486578822135925, -0.08258997648954391, -0.22945009171962738, -1.4975402355194092, 0.8762234449386597, 1.3469411134719849, -0.2643009126186371, -0.6334601044654846, -0.18790581822395325, -0.686759352684021, 1.092476725578308, 0.14519935846328735, 1.1543906927108765, 0.3036288022994995, 0.770694375038147, -0.9962579607963562, -0.5719664692878723, 1.1610149145126343, 0.1534050554037094, -0.7253968119621277, -0.26891544461250305, 0.22608253359794617, -1.3798801898956299, -1.4638137817382812, 0.9748170375823975, 0.12276080250740051, 1.0983012914657593, -1.5121394395828247, -1.9477787017822266, -0.08580984175205231, 1.4456217288970947, -0.183467298746109, 0.1685117930173874, 0.5990627408027649, -0.4955994784832001, 0.2568355202674866, 0.9477844834327698, -0.4697935879230499, -1.6852121353149414, 0.6865072846412659, -2.290907621383667, -0.8386443257331848, -1.4448342323303223, -0.47258782386779785, -1.9376415014266968, -1.0588963031768799, 0.48555853962898254, -0.8853644132614136, -0.9982538223266602, -0.6800809502601624, 1.8081626892089844, -2.2787232398986816, -0.24946723878383636, 0.42542609572410583, -0.39978617429733276, 2.138112783432007, -0.7372567057609558, 0.28874441981315613, -0.5413826704025269, 0.6393488645553589, -0.013768782839179039, -0.113804891705513, -0.30900853872299194, -0.5178508758544922, 0.42140698432922363, -0.7474502325057983, 2.178820848464966, -0.4898544251918793, -0.7462872266769409, 0.24066723883152008, -1.1609169244766235, 1.1008360385894775, -0.5031195878982544, -0.7302714586257935, -0.6639911532402039, 1.3418021202087402, -1.2069975137710571, 0.5264720916748047, -0.5309662818908691, 0.26709550619125366, -1.4271715879440308, -0.8267771601676941, -0.9040890336036682, 0.23214440047740936, -1.0069622993469238, 1.1061432361602783, 0.8087754249572754, 1.1408991813659668, -0.9833489656448364, -0.11828576773405075, 0.15947550535202026, 0.623476505279541, 0.16194304823875427, -0.32586777210235596, 0.7060055732727051, -0.0893002450466156, 0.3539941608905792, -0.5876783728599548, -1.2374635934829712, -0.4472730755805969, 1.0674090385437012, -1.5845743417739868, -0.09755446761846542, -1.128859519958496, -0.6444594860076904, -1.0087636709213257, 0.9095357060432434, 1.2003182172775269], [1.555915117263794, 1.099988341331482, 0.843412458896637, 0.07249331474304199, 0.031852398067712784, -0.3434303402900696, -0.8908693790435791, -0.10490751266479492, -1.6061975955963135, 0.27300524711608887, 1.2470847368240356, 0.7994047999382019, -0.22961944341659546, 0.08203700184822083, 1.3268115520477295, 0.0996612161397934, -0.11980585008859634, -1.16359281539917, 2.4644904136657715, 0.28040042519569397, -0.984052300453186, 2.426417350769043, 0.5112362504005432, -0.2972709536552429, -2.0240132808685303, -1.4944418668746948, 1.0462243556976318, -2.0597777366638184, 0.9557992219924927, 1.4309847354888916, -1.5812784433364868, -1.6291064023971558, 0.9137051701545715, -0.9697567820549011, 0.9062036275863647, -2.2251272201538086, -0.06601642072200775, 2.258528709411621, -0.060602519661188126, -1.230991244316101, -0.4610356390476227, 1.412035584449768, 1.4961613416671753, 0.3189966380596161, 0.25097760558128357, 0.11633771657943726, 1.5785373449325562, 0.30416426062583923, -1.2280771732330322, 1.4209998846054077, 0.8329735994338989, -0.6251046657562256, 0.22655807435512543, 0.02100084163248539, -0.8313737511634827, 0.5834597945213318, 0.7268983721733093, 0.2791084349155426, 1.139343023300171, -1.9944736957550049, -2.149643898010254, 0.7021200656890869, 1.1294833421707153, -0.8268762230873108, -0.10832453519105911, 0.8091230988502502, 0.10195357352495193, -1.2643193006515503, 1.0582960844039917, 1.8472661972045898, -0.610568642616272, 1.1519380807876587, -2.256058692932129, 0.06667978316545486, -0.2241794914007187, 0.31701305508613586, -1.0262892246246338, -0.6912892460823059, -0.45344454050064087, -0.030963286757469177, 1.4375226497650146, 0.8311070203781128, 0.7460849285125732, -0.3708593249320984, -0.22090613842010498, -0.7981336712837219, 2.463141441345215, -0.5633887648582458, 0.8233181834220886, -0.1459224373102188, -0.8790102601051331, 0.13309863209724426, 0.6763283014297485, 1.1363178491592407, 0.7148346304893494, -0.6109151840209961, 0.8310819268226624, -0.9616113305091858, 1.0449614524841309, 0.3164227306842804, 1.0759046077728271, -1.1001718044281006, -1.6941673755645752, 1.3721002340316772, 0.8498911261558533, -1.1274877786636353, 0.7270005941390991, -0.9570876359939575, 1.1042879819869995, -0.06323809176683426, -0.7389033436775208, -2.016148090362549, -1.0867319107055664, -1.4895225763320923, -0.9628359079360962, -0.9926745295524597, -2.77333402633667, 0.2536149024963379, 0.8721231818199158, -1.6378082036972046, -0.04434388875961304, -0.11729045957326889, -0.8406936526298523, 0.690787136554718, 1.468910574913025, -1.0322107076644897, -1.0716311931610107, -0.23242713510990143, -1.6180552244186401, -0.1973896473646164, 0.5987862348556519, 0.37234237790107727, 0.9690345525741577, -0.6697643995285034, -1.2173339128494263, -1.9331742525100708, -0.43153247237205505, 1.8628615140914917, -2.3774757385253906, -0.749093770980835, 0.011233801953494549, -0.6942427158355713, -0.17477723956108093, -2.3575925827026367, -0.7382450699806213, -2.050226926803589, -1.4829853773117065, -1.0963143110275269, 0.4142722189426422, 0.029639199376106262, -0.37178295850753784, -1.169203519821167, -0.04115697741508484, -0.09950046986341476, 0.99079430103302, -0.523252010345459, 0.2355596274137497, -0.13649004697799683, -1.1052496433258057, 2.72196102142334, 0.01705215871334076, -2.255014181137085, -0.6105038523674011, 1.9274013042449951, 2.027750253677368, 0.3932841420173645, -2.217282772064209, 0.379576712846756, -1.0180127620697021, 0.9960208535194397, -0.06883082538843155, -0.05075208842754364, 0.23568052053451538, 0.02680983394384384, -0.10953935235738754, -0.46375370025634766, -0.18180114030838013, -0.046651702374219894, -1.259543538093567, -1.4987504482269287, 1.5557750463485718, -0.49789026379585266, 0.35136839747428894, -0.3856826424598694, 0.24936698377132416, -1.4987645149230957, 1.3794512748718262, -0.10039925575256348, 0.5594782829284668, 0.5873381495475769, 1.0884815454483032, 2.911097288131714, -1.0670018196105957, 0.75303715467453, -0.2513505220413208, -0.33183547854423523, -0.04665759950876236, -1.0158789157867432, -0.7026317119598389, -0.29541653394699097, 0.4761134088039398, -0.7286240458488464, -0.6858577132225037, -0.46167293190956116, 1.2568339109420776, 1.165210485458374, 1.1822727918624878, -0.23338285088539124, 1.5185383558273315, -1.9811551570892334, -0.7581700086593628, 0.18011517822742462, 0.6658473610877991, -1.9710379838943481, -0.7319456934928894, 0.4384934604167938, -0.3859075605869293, 3.013125419616699, 1.3685392141342163, 0.9405845999717712, 1.2795557975769043, -0.4437369704246521, -1.1526026725769043, -0.06813009083271027, -0.29766836762428284, 1.0176053047180176, -0.21620452404022217, -0.6112439036369324, -0.06583520770072937, -0.8954125642776489, -0.9258082509040833, 0.3562992215156555, -0.5929920077323914, 0.4400995075702667, 1.4264310598373413, -2.186201333999634, -0.5246927738189697, 1.2674055099487305, -1.3867902755737305, -0.494912326335907, -1.0233842134475708, -0.07152212411165237, 0.5846456289291382, 0.7993234395980835, 0.7278898358345032, 1.2336615324020386, -1.0356203317642212, -0.2494632601737976, 0.31963908672332764, 0.01859309896826744, 0.13091430068016052, -0.4721647799015045, -0.9705690145492554, 1.0041019916534424, 1.2283552885055542, 1.9455335140228271, -0.19567988812923431, -1.1126909255981445, 1.161172866821289, -0.40166524052619934, 0.7570667862892151, -0.9180300831794739, 1.1888504028320312, -0.15430662035942078, -0.2474866658449173, 0.5041937828063965, 0.5423102378845215, 0.47301027178764343, -1.3143213987350464, -0.6944813132286072, -0.5898563861846924, 0.14746318757534027, -0.555352509021759, 0.24731099605560303, 0.18892532587051392, 0.7907110452651978, 0.08104564994573593, -0.37400975823402405, 0.8915854692459106, 1.3709704875946045, -0.4766331613063812, 4.1480488777160645, 0.06380749493837357, -0.7611076831817627, -0.1557304859161377, -0.8957874178886414, -0.6872024536132812, 1.1596218347549438, -1.2080167531967163, -0.7958786487579346, -0.6576563715934753, -1.5365394353866577, 0.40854859352111816, 1.2799569368362427, -0.3015490770339966, 0.7998822331428528, 0.6678915023803711, -1.0781044960021973, -0.47329485416412354, 0.4460354149341583, 0.7771804332733154, 0.28220483660697937, 0.2496555894613266, -1.0820387601852417, -1.737754464149475, 0.6762018799781799, 0.9172536730766296, 1.104013442993164, -0.49540528655052185, 0.19414876401424408, -0.5717700719833374, 0.5111935138702393, -1.857414960861206, 0.3349042236804962, 0.10104170441627502, 0.7380861639976501, 0.30315035581588745, 0.2522329092025757, 0.10684959590435028, 1.4605883359909058, -0.7232558727264404, -0.8316508531570435, -1.0079624652862549, 0.619530200958252, 0.3964303731918335, 2.451446056365967, 0.011018780060112476, -0.5494418740272522, -1.3161230087280273, -0.21807961165905, 1.7761512994766235, 0.39007604122161865, 0.16147007048130035, -1.2999523878097534, 0.07257863879203796, 0.7891278266906738, 0.176114022731781, -0.7583317160606384, -0.2592056691646576, -0.11000798642635345, 0.91902095079422, -2.0706968307495117, 1.881127953529358, 0.1711425632238388, -1.5428563356399536, 0.2896522879600525, 0.8942541480064392, 0.37841421365737915, -0.009400433860719204, -0.48170700669288635, 0.0616784542798996, 0.011673794128000736, 2.2773983478546143, 1.0803980827331543, -0.8247260451316833, 1.542681336402893, 0.6087145209312439, 0.5042328834533691, 0.4397242069244385, 0.49355024099349976, -0.2432185411453247, 0.3537956476211548, -0.2763376533985138, -0.03919930011034012, 0.43861058354377747, -0.8496131896972656, -2.173650026321411, 1.2038377523422241, -0.14707258343696594, 1.0855916738510132, 1.2167997360229492, 0.6195421814918518, 0.9450619220733643, -1.0617595911026, 0.4195508360862732, 0.9279786348342896, -0.20006869733333588, -1.4021021127700806, 0.7186440825462341, 0.7924549579620361, -0.5011942386627197, 1.098638892173767, -0.21465449035167694, -1.6564583778381348, 0.331096887588501, 0.5756849646568298, -1.50110924243927, 0.4328925609588623, 0.8880932927131653, 0.24909155070781708, -0.9182218909263611, 1.7312533855438232, 1.6140732765197754, -1.1199991703033447, -3.375286817550659, -1.705393671989441, -0.2974524199962616, 0.2500034272670746, -0.06051569804549217, 0.5639611482620239, -2.1738345623016357, -1.11246657371521, -0.5024861097335815, -0.9693098068237305, -1.7352596521377563, -0.42042791843414307, 0.6976994276046753, -0.7126638293266296, 1.1395050287246704, 0.4754554331302643, -1.6925231218338013, 1.8807759284973145, 0.35698825120925903, 2.4581356048583984, -0.3951900601387024, 0.6139692068099976, 0.20829427242279053, -0.29481402039527893, -0.285648375749588, 0.22031261026859283, -0.2324932962656021, 0.25837406516075134, -0.46453824639320374, 0.8038464188575745, 0.8021343350410461, 0.2664092779159546, -2.2896323204040527, -0.2954657971858978, -0.5692028403282166, 0.6077947020530701, -0.022911876440048218, 0.22198651731014252, 1.2765414714813232, 1.2193584442138672, 1.4031318426132202, -0.47610846161842346, -0.5521818995475769, -0.9528028964996338, -0.17775128781795502, 0.6560608148574829, -0.5810623168945312, -0.6312973499298096, -0.6990652084350586, 0.8126024007797241, -1.211104393005371, 0.791501522064209, -1.8821954727172852, 0.43047335743904114, -0.48204120993614197, 1.2845852375030518, 0.29361510276794434, 0.5108199119567871, -2.4163761138916016, -0.5285398364067078, 0.6205577254295349, 0.7902477383613586, 1.428969383239746, 1.1106171607971191, -1.0535968542099, -0.2043500393629074, -1.5764250755310059, 0.7324017286300659, 0.9792974591255188, -0.7296644449234009, -1.140251874923706, -0.33075323700904846, -1.1682480573654175, -1.2128348350524902, -0.7585546374320984, 0.5079546570777893, -1.0532816648483276, 0.07516704499721527, 0.7046388387680054, 0.33159253001213074, -1.5354948043823242, -1.666184902191162, 0.24132350087165833, -0.03313037008047104, -0.4210983216762543, 0.9208826422691345, 0.8153310418128967, -0.6455925107002258, -1.023069143295288, 0.7616080641746521, 0.47969555854797363, -1.6781666278839111, -0.23430699110031128, 0.8311337828636169, 0.28374183177948, 0.33650466799736023, 0.1614205539226532, 0.5852447748184204, -0.28006306290626526, -0.24047552049160004, 0.2371019572019577, -1.0548111200332642, 0.26401659846305847, -1.052987813949585, 0.10220129042863846, -0.1383633017539978, 0.5828922986984253, 0.6512709856033325, 1.3270213603973389, -1.1895482540130615, 0.9050961136817932, 0.39601147174835205, -0.04943099990487099, -1.3504087924957275, 2.073323965072632, 0.20687253773212433, -1.5225716829299927, 0.042077962309122086, -2.515911102294922, 0.23324249684810638, -1.0067775249481201, -0.422450989484787, -0.3026333153247833, 1.1153347492218018, -1.3151824474334717, -0.6309424638748169, -0.09279517084360123, -0.4310202896595001, -0.9032245874404907, 0.551517128944397, 0.09719541668891907, 0.060584526509046555, 0.5974895358085632, -0.7254956364631653, 0.05003269389271736, 0.9017719030380249, -0.6068023443222046, -0.2191910743713379, 0.05123775824904442, 0.8460814356803894, -1.8110309839248657, 1.475144863128662, -0.5331274271011353, 1.0141973495483398, 2.40712833404541, -0.533454954624176, 1.8066089153289795, -0.11266426742076874, -3.661142110824585, -0.6027098298072815, -1.1045485734939575, -0.6671654582023621, 1.2349110841751099, -1.3861578702926636, -0.5014006495475769, -0.3187766969203949, 0.24065473675727844, -0.5122933983802795, 2.6316845417022705, -0.4077751040458679, 0.23615537583827972, 1.1076130867004395, -1.5572571754455566, -1.5605424642562866, -1.8023689985275269, 0.7448095679283142, 0.6589191555976868, 1.8210879564285278, -1.0578817129135132, 0.15574268996715546, -1.9086953401565552, 0.21376429498195648, 1.0020793676376343, 1.6353180408477783, -0.2893383800983429, -1.0041977167129517, 0.9739224910736084, 0.6479822993278503, -1.802369236946106, -0.15040701627731323, -0.8471287488937378, 0.5867513418197632, -1.0486352443695068, -0.7287833094596863, 0.5859747529029846, -0.38427630066871643, -1.1900113821029663, 0.2867860496044159, 0.5036330223083496, -0.7988067269325256, 1.0783861875534058, -0.7012293338775635, -0.3523981273174286, -0.7862161993980408, -0.24781924486160278, 2.185969114303589, -1.8359696865081787, 0.5728159546852112, 0.8306558728218079, -0.28900444507598877, 0.5628367066383362, -0.28730612993240356, -0.8246811032295227, 0.9581301808357239, 1.016581416130066, -1.1745574474334717, 0.9692366719245911, -0.07525008916854858, 1.210677146911621, -0.9955562949180603, 1.4564557075500488, -1.4437687397003174, 1.0155737400054932, 0.5058438777923584, 0.2504850924015045, -0.5329344868659973, 0.5848007202148438, 0.17664296925067902, 1.5387133359909058, -1.6637243032455444, -0.11726133525371552, -1.642775297164917, 0.1431911587715149, -2.1677069664001465, -0.4012354016304016, -1.7588380575180054, -0.6035235524177551, 0.4389502704143524, -0.3370874524116516, 0.4912150800228119, -0.13192786276340485, 0.11423827707767487, -0.2583256959915161, -0.322979599237442, 2.5412113666534424, -0.21101143956184387, 0.7288914322853088, 2.4002251625061035, -0.37960386276245117, -0.0868830680847168, -1.0171524286270142, 0.4331287443637848, -0.5981545448303223, 2.309997081756592, -0.49499353766441345, 0.8783475756645203, 1.3637182712554932, 1.558372139930725, 0.2440125048160553, 0.3814961016178131, -1.3591769933700562, 0.10287593305110931, -0.37251198291778564, -1.2397373914718628, 1.9748033285140991, -0.5871216654777527, 0.08395085483789444, -0.08140041679143906, 0.7811038494110107, -0.26000717282295227, -1.0476469993591309, 0.07327007502317429, 1.4940747022628784, 0.003518796991556883, 2.7108919620513916, -1.042321801185608, 1.0887876749038696, 0.14942428469657898, -0.4031479060649872, -0.9044204950332642, 1.1424975395202637, 1.2815513610839844, -0.8123791217803955, -0.1884520798921585, -0.7696682810783386, -0.29997316002845764, -0.7550976276397705, -1.5435456037521362, 0.5668343901634216, 0.03719094395637512, 0.844977617263794, -1.3470772504806519, -0.5827659368515015, -0.2847253680229187, -1.2664799690246582, 0.7732224464416504, -1.2104980945587158, 2.2614195346832275, 0.11093117296695709, 1.761388897895813, 0.7898417711257935, 0.850324273109436, -0.4904220402240753, 2.1720376014709473, -0.24421678483486176, -0.009381038136780262, -0.9596858620643616, -0.7134756445884705, -0.3303271234035492, 1.4238227605819702, 1.3882601261138916, -1.889763355255127, 1.9104101657867432, -1.1971864700317383, 0.6709303259849548, 0.37766891717910767, 0.10444393754005432, 0.06339838355779648, -1.2902361154556274, 0.3329359292984009, -0.20769202709197998, -0.25882580876350403, 0.5965207815170288, -1.8162542581558228, -1.1336592435836792, -0.526823878288269, 1.8199553489685059, -0.5663637518882751, 0.5421392917633057, -1.3801143169403076, 1.1849186420440674, 1.0156313180923462, -0.18218065798282623, 0.39955100417137146, -0.9052280187606812, -0.7787081003189087, 1.0030691623687744, 0.40167051553726196, 0.26654955744743347, -0.5089515447616577, -0.7546547055244446, 1.8708100318908691, 2.095872402191162, 0.7350367903709412, -0.08018016070127487, 0.9352005124092102, -0.7065305113792419, 0.08413363993167877, 0.23676680028438568, -0.38124433159828186, 0.24687615036964417, 0.9258068203926086, -1.2758197784423828, 2.3983945846557617, 0.34291592240333557, 0.12401866912841797, 0.04116862639784813, -1.7602410316467285, 0.2333594709634781, 0.5184819102287292, -0.7358435988426208, -0.8570988774299622, -1.2792048454284668, 0.398250937461853, -0.22295896708965302, 0.1908731907606125, -0.7601138353347778, -1.4869945049285889, -0.318753719329834, 1.8966013193130493, 0.9890011548995972, 0.49248799681663513, -0.5284767150878906, 0.1537410020828247, -0.429980605840683, -1.3393160104751587, 1.2685120105743408, 1.4247417449951172, 1.2857418060302734, -0.5007449388504028, -0.1775815188884735, 0.5454424023628235, 1.425702452659607, 1.1442302465438843, 0.949998676776886, 1.2463148832321167, 0.7803001999855042, 0.030375119298696518, -1.3684958219528198, 0.635590136051178, -0.2524305582046509, 0.1883155107498169, 0.40140506625175476, -0.053477268666028976, 1.5562293529510498, 1.0416990518569946, -2.28987979888916, 1.0892592668533325, -0.16684490442276, -0.12996134161949158, -1.3015652894973755, -0.7629591226577759, 0.9725431799888611, 0.921138346195221, 0.9149818420410156, -0.3654493987560272, 1.1110812425613403, 1.5321239233016968, 0.20568424463272095, 0.18978415429592133, -2.2735435962677, -1.412591576576233, 0.19145667552947998, -2.485398292541504, -0.6139894723892212, 1.4439277648925781, -1.4368023872375488, -2.1577603816986084, 0.010763099417090416, 0.07576645165681839, -0.3653760850429535, -0.4543561637401581, -0.4416481554508209, 0.2559833526611328, 0.271329790353775, -0.013329229317605495, -0.5649673342704773, 1.7417702674865723, -1.2595720291137695, -0.05322936922311783, 0.0720365047454834, 0.34303799271583557, 0.6217277646064758, -0.4493280053138733, 0.5278093814849854, 0.35722658038139343, 0.18420781195163727, 0.7366001605987549, 1.0433363914489746, 1.7121708393096924, -0.6246396899223328, -1.112909197807312, -0.15284208953380585, -0.7724547386169434, 0.7111613154411316, -0.4477926790714264, -0.8792850971221924, -0.9453446269035339, -0.7779797315597534, -1.0704740285873413, 0.7403069734573364, -1.7345056533813477, 0.8303672671318054, -1.8130834102630615, -0.8352123498916626, 0.16095630824565887, 1.8436814546585083, -0.8129790425300598, -0.07534550875425339, 0.32373279333114624, 0.38620099425315857, -2.332590103149414, -0.4918331503868103, 1.9237167835235596, -1.286781907081604, -0.1657349318265915, 0.31828799843788147, -1.4454455375671387, -0.4175042510032654, 1.589020848274231, 1.9069552421569824, 1.091291069984436, 0.2974247336387634, 0.26650184392929077, -0.1760440319776535, 1.119529366493225, 2.319444179534912, -2.4561684131622314, -0.7331091165542603, 0.959379255771637, 0.032996345311403275, 0.6526488065719604, 0.2971706986427307, 1.5298298597335815, -0.20297962427139282, 1.2952009439468384, 1.8193212747573853, -1.7254307270050049, 0.937986433506012, 0.4962734580039978, 0.6497370600700378, -0.5619982481002808, 1.0236725807189941, 0.4081396460533142, -0.11993670463562012, -1.3738813400268555, 1.1597816944122314, -0.45070523023605347, -0.6904443502426147, -2.0380163192749023, 0.31931665539741516, 0.019006622955203056, -1.0710498094558716, 0.371491938829422, -0.6528909206390381, -0.6687320470809937, 0.3634355366230011, 1.2263888120651245, -1.6021828651428223, -0.08448993414640427, -0.25952255725860596, -0.19524399936199188, -0.2675355076789856, 0.5397077798843384, 1.2654750347137451, -0.825667142868042, -1.5494745969772339, -0.13398079574108124, -0.7019282579421997, 0.11401179432868958, -1.2397964000701904, -0.09305798262357712, -0.647347629070282, 0.5049114227294922, -0.0692036896944046, 0.35483479499816895, 0.4973728358745575, 0.6862777471542358, -0.008120432496070862, -1.8518511056900024, 0.29968374967575073, 1.8048062324523926, -1.2936515808105469, -0.6039537191390991, 0.3960125148296356, 1.7830543518066406, 0.24414198100566864, 0.6396703124046326, -0.3078140914440155, -0.1656343787908554, 0.1774677187204361, -0.503792941570282, 1.0210247039794922, 1.2131311893463135, 0.4589616060256958, -0.5931745171546936, -1.6201368570327759, -0.1097576692700386, 0.38374063372612, -0.5038387179374695, -0.8078334927558899, 0.6206159591674805, -1.6915775537490845, -0.0628352090716362, -0.6726225018501282, 0.3821418285369873, -0.004583013243973255, -0.3669465184211731, -0.5974846482276917, 0.3749631345272064, -0.42806491255760193, 0.5772281885147095, -0.7324769496917725, 1.1175081729888916, 0.19811606407165527, -0.9419192671775818, 2.2240095138549805, 0.3057304620742798, 2.409372329711914, -1.0619858503341675, 0.3391767144203186, -0.2218751460313797, 0.3526691198348999, -0.7507443428039551, 0.7689647674560547, -0.3572940230369568, 0.013888356275856495, -0.735276460647583, 0.04719018563628197, -1.357693076133728, 0.13384586572647095, -0.5593523383140564, 1.6388558149337769, -1.117436408996582, 1.9110243320465088, -0.37223318219184875, -0.6713675260543823, -0.8239756226539612, -0.10846208781003952, 0.9414686560630798, 0.3223760724067688, 0.2664688229560852, 0.22345471382141113, 0.5555936694145203, -0.5813665986061096, 1.8867590427398682, -0.1612856686115265, 0.07826974987983704, 1.0190598964691162, 1.3924756050109863, -1.4494613409042358, 0.8349124193191528, -1.4841549396514893, 0.07255276292562485, -0.13941140472888947, -0.3637245297431946, 1.4256876707077026, -0.5578306317329407, -0.11763953417539597, -0.9209170937538147, -1.1421750783920288, 1.1308714151382446, -0.5296552181243896, 0.6375957727432251, -0.28280892968177795, 0.5770209431648254, 0.06186504289507866, 1.5899956226348877, -0.5166478157043457, 0.05593307688832283, -0.9857906103134155, -1.0157595872879028, 2.628195285797119, -1.6150771379470825, -0.7049046158790588, 0.22608229517936707, -1.0415524244308472, -1.4395045042037964, -0.8076901435852051, 0.17380717396736145, 0.23670373857021332, -2.389409303665161, -1.2414942979812622, -0.9617069959640503, -0.9882190227508545, 0.025034809485077858, -0.8431028723716736, -0.341670960187912, -1.1473524570465088, 0.9015493392944336], [0.08038341999053955, -0.35785508155822754, 0.14834994077682495, 0.25683537125587463, 0.35125792026519775, -2.424110174179077, -1.1319106817245483, -0.5176479816436768, 0.24229216575622559, -0.0851556584239006, 1.2797186374664307, 0.6671596765518188, -0.6674632430076599, -1.360473871231079, 0.15526776015758514, 0.043016258627176285, -0.4184637665748596, 0.06471305340528488, 0.10832313448190689, 1.5623971223831177, 0.9183782339096069, -0.15298433601856232, 0.06332503259181976, 1.122180461883545, -1.5590691566467285, 0.5261347889900208, -0.46552276611328125, -0.7677662968635559, 0.821730375289917, 1.939026951789856, 0.7622668147087097, 0.018730076029896736, -1.2578097581863403, -0.0004729227221105248, 0.048952341079711914, -1.1367450952529907, -1.0219156742095947, 2.0876779556274414, 0.9760212302207947, 0.05026448890566826, -0.013198312371969223, -1.4224209785461426, -0.9556601643562317, -0.10117150843143463, 0.7685523629188538, 0.8695085048675537, -1.8615069389343262, 1.1880073547363281, 0.4630294144153595, -0.016722925007343292, -0.831835150718689, -2.06196928024292, -0.8053122162818909, -0.3476889729499817, -1.6816182136535645, 0.37696465849876404, 0.3110186755657196, -2.5473406314849854, 0.5390775799751282, -0.07896625250577927, -0.28449901938438416, -0.4873427152633667, 0.6644524931907654, 1.5922523736953735, 1.7650212049484253, -0.5814902782440186, -1.4059499502182007, -0.9628875255584717, -0.3485978841781616, 1.5688527822494507, 0.7057781219482422, 0.48439648747444153, -2.102006673812866, -0.1718633621931076, -0.3351871371269226, 0.3924081325531006, -1.8746572732925415, -0.3137848675251007, -0.974609375, -0.9981744289398193, 0.45998331904411316, 0.6102807521820068, -1.3027807474136353, -0.16635891795158386, -0.27708300948143005, 2.307589292526245, -1.320046067237854, 0.288775235414505, -0.578751266002655, 0.6581661105155945, -0.39240512251853943, 1.4367209672927856, -0.17175142467021942, 0.26468437910079956, 1.560881495475769, 0.13795505464076996, -0.03200443834066391, 0.31111225485801697, -0.16621260344982147, 0.384023517370224, 0.21442469954490662, -0.3767941892147064, -0.6920778751373291, 0.6149835586547852, 0.05267160013318062, 0.6293482184410095, -2.3675124645233154, 0.5923134088516235, -0.9231619834899902, 0.5948666930198669, 0.36309725046157837, -0.33214014768600464, -0.14681541919708252, 0.4264812171459198, 1.2466561794281006, 0.6811280250549316, 0.5718770027160645, -0.06347797065973282, -0.6889702081680298, -0.8027287125587463, 1.436599850654602, 0.6301135420799255, -0.05104455724358559, 0.571017861366272, 0.3483816385269165, -0.23191364109516144, 1.695297122001648, 0.23289424180984497, 0.5648401379585266, 0.009520943276584148, -0.8078523278236389, 1.7612388134002686, 0.3771195709705353, 0.044959116727113724, 0.7459736466407776, -1.9340386390686035, 0.3908040225505829, -0.803187906742096, -1.029779076576233, -0.20538726449012756, 0.1044173464179039, 0.8029844164848328, 0.24019083380699158, 1.4674128293991089, 0.38588401675224304, -0.1232801303267479, -0.12417756021022797, 0.0886939987540245, -0.396073579788208, -0.5531778335571289, -0.6253036856651306, -2.425773859024048, 0.49673721194267273, -0.17893196642398834, 0.8059886693954468, -0.03488236665725708, 1.0315577983856201, -0.05221859738230705, -1.2151620388031006, -0.4378848671913147, 0.11546090990304947, 1.5882346630096436, -1.158704161643982, 0.2677980959415436, 0.8700515627861023, 0.6297926902770996, -0.7831058502197266, -0.8234473466873169, 0.516357421875, 0.7700071930885315, 0.5850862264633179, -0.10239288955926895, -0.8035786747932434, -0.16943331062793732, -0.8034895062446594, 0.4981864392757416, -0.5252317786216736, -1.9831260442733765, -0.9742831587791443, 0.30063357949256897, -0.0534588061273098, 0.07927633076906204, -0.006738229189068079, 2.0576977729797363, -0.005252002738416195, -0.4635266661643982, -0.4809165298938751, -0.2519327402114868, -1.9710999727249146, -0.6658150553703308, -0.9354250431060791, 0.43480703234672546, -0.5555760860443115, 0.47359341382980347, -2.209038257598877, 0.6430371999740601, 0.655232846736908, -1.8440721035003662, 1.4982860088348389, 0.40271782875061035, -0.338722288608551, -1.4631322622299194, -0.8350384831428528, -1.7556657791137695, -0.77698814868927, -1.4469366073608398, -1.1216686964035034, -0.35642144083976746, -2.5412235260009766, 0.31008076667785645, -0.13055811822414398, 0.2719322144985199, -0.8017141222953796, 0.5107867121696472, 2.1140542030334473, -1.3096286058425903, -0.9457324743270874, -1.5169154405593872, -1.8557424545288086, -2.1371941566467285, 0.6029510498046875, 0.18130609393119812, 0.24472330510616302, 1.146653175354004, 0.372313916683197, -0.7597230672836304, 0.12612131237983704, 0.7922560572624207, -0.9566264152526855, -0.47777003049850464, 0.8122007846832275, 0.8005298972129822, 0.30585241317749023, -1.3205065727233887, 0.7948421239852905, -0.18366144597530365, -1.9603830575942993, 0.6321030259132385, -0.35611969232559204, 1.948607087135315, -1.4821913242340088, 1.180910587310791, -0.5785101652145386, 0.3402186930179596, -0.42094355821609497, 0.3276662528514862, -2.36767840385437, 0.7063179612159729, -1.297611951828003, -2.507266044616699, 0.2487768977880478, -0.21317942440509796, -1.3453247547149658, -0.19508710503578186, -0.92336505651474, -1.0894076824188232, 0.3684477210044861, 2.559422016143799, 0.034725118428468704, -1.7571815252304077, 1.5446557998657227, -0.09883026778697968, -1.8622617721557617, -0.21770277619361877, -0.042135223746299744, 0.575313925743103, -0.2749287188053131, -0.38174983859062195, -0.910140335559845, 0.7785196304321289, -1.5266010761260986, -1.0217255353927612, 0.15637759864330292, -0.6294533610343933, 0.6883072257041931, 1.6025439500808716, 0.3021465241909027, 0.7939440608024597, -0.42315736413002014, 1.9754927158355713, -0.6735865473747253, -0.15820543467998505, -1.1927138566970825, 0.8556685447692871, -0.41507813334465027, 1.5779660940170288, -0.2849291265010834, -0.9666829705238342, -1.0234358310699463, -0.2199535369873047, -0.18750125169754028, 0.30312976241111755, 0.07177506387233734, 0.7721590995788574, -0.053621284663677216, 1.9470237493515015, -0.372381329536438, -0.7542214393615723, -1.0088831186294556, -0.0072463396936655045, -0.06881487369537354, 0.16296017169952393, 0.16981767117977142, -0.17884236574172974, 0.26039713621139526, 0.10460430383682251, 0.6009083986282349, 0.36412057280540466, -1.0763005018234253, 0.6791110634803772, 0.10509859025478363, -0.3141372799873352, 0.5868165493011475, 0.958282470703125, -0.20903849601745605, -2.0501341819763184, 0.7006037831306458, 0.3891933858394623, -0.7206626534461975, -0.063698410987854, -0.06935951113700867, 0.9076659679412842, -0.9259495139122009, 0.3451350927352905, -1.1003903150558472, -1.027248740196228, -2.2874910831451416, 1.0170681476593018, 0.43005508184432983, -0.01896616257727146, 0.6790433526039124, 0.13424396514892578, -1.0204576253890991, -0.8565052151679993, 0.5496271252632141, 1.437913179397583, -0.50942063331604, -0.8449322581291199, -0.38117527961730957, -1.1586812734603882, -0.05387352034449577, -2.180799961090088, 0.8350170254707336, 0.5394731760025024, 0.9531881213188171, 0.46919864416122437, 0.030392305925488472, -0.9366229772567749, -0.44536998867988586, 0.17799000442028046, -0.10153017193078995, -2.504206657409668, 0.11929939687252045, -0.5216198563575745, -0.5939381718635559, 0.7022048830986023, 1.6645424365997314, -0.13470356166362762, 0.508880615234375, -0.09126030653715134, 1.957261085510254, -0.046173978596925735, -0.7568004727363586, 0.3698452115058899, 0.4727306663990021, 0.326630562543869, -0.8251566290855408, -0.7532616257667542, -0.8230404257774353, 1.2242860794067383, -0.37290242314338684, 0.2890174984931946, 1.2425813674926758, -1.0143139362335205, 0.6734787821769714, 0.30740389227867126, 1.3431551456451416, 0.5714753866195679, -0.02112772688269615, -0.4988822340965271, 0.6755562424659729, 0.8260821104049683, -0.5741828083992004, 0.8647805452346802, 0.6694409847259521, -1.5758124589920044, 0.3740617036819458, -0.145053893327713, 0.7698045969009399, -2.5663061141967773, -1.684545636177063, -1.2091898918151855, 1.6051722764968872, -2.606959581375122, 0.7914999723434448, -0.22983458638191223, 0.19870561361312866, 0.94284588098526, 0.4106704294681549, -2.0209715366363525, -0.8784649968147278, -0.4023994207382202, -0.39549368619918823, -0.727252185344696, 0.9275765419006348, -0.3429534435272217, 0.033447034657001495, -0.6420961022377014, 1.110635757446289, -0.6855291724205017, -1.350448489189148, -2.481707811355591, -0.7247136235237122, 0.2715228497982025, -0.7819337844848633, 0.2168934941291809, -0.05372191220521927, -0.7257644534111023, 1.0290156602859497, -0.2779845893383026, 1.5594737529754639, -0.22224624454975128, -1.8638120889663696, 1.2371724843978882, -2.5202343463897705, 1.7506135702133179, 0.6579079627990723, -1.1265463829040527, 0.46826720237731934, 0.013130728155374527, -0.17969565093517303, -1.16329026222229, -0.5844085216522217, -1.5613908767700195, -0.22766366600990295, -0.2660885155200958, 0.3500956594944, 0.27909228205680847, 1.435260534286499, 1.6620100736618042, -0.1504003405570984, -1.0728296041488647, 0.474443644285202, -0.845845103263855, 1.6063148975372314, 1.267967700958252, -1.4735642671585083, 0.8485887050628662, 0.14328144490718842, -2.5538980960845947, -0.0619560107588768, -1.615456461906433, 1.0091358423233032, 0.5721166133880615, 1.4933687448501587, 0.3434373438358307, -2.710200786590576, 1.3089706897735596, 0.04156990721821785, -0.627704918384552, -1.0230385065078735, -1.6932274103164673, -1.215651273727417, 0.6196897625923157, -1.1704305410385132, 1.2021713256835938, 0.14668293297290802, 0.4057885408401489, -0.9307764172554016, 0.19510148465633392, 0.6757246255874634, 1.3635075092315674, 1.2471448183059692, -0.4121536314487457, -0.19075128436088562, -0.2931252717971802, -1.281681776046753, 0.10065558552742004, 0.3100864887237549, 0.16728979349136353, 1.6272571086883545, -1.6927660703659058, -0.2251223623752594, -1.3909651041030884, 1.178292989730835, -0.8974226117134094, 0.020501254126429558, -1.4393290281295776, -0.1709890365600586, 0.3089044988155365, 0.4317994713783264, -2.154204845428467, 0.7850590944290161, -0.999403178691864, -2.4405436515808105, -0.1189841702580452, 0.48066291213035583, 1.7024480104446411, -0.34399476647377014, 0.5534344911575317, -1.075770616531372, -0.08895055204629898, 0.6008861660957336, -0.34386733174324036, 2.2145965099334717, 0.26794248819351196, -1.1192692518234253, -1.0159610509872437, -0.4235724210739136, -0.675977885723114, 1.7212786674499512, -0.22568100690841675, -1.0724327564239502, -0.6474814414978027, -0.2444186508655548, 0.8769784569740295, 0.35533177852630615, 0.1623999923467636, -0.7629507780075073, 0.3753430247306824, -0.0797240138053894, -0.7070372700691223, 1.0813478231430054, -1.6372780799865723, 1.1746346950531006, 1.728844165802002, -0.9353924989700317, 0.038634758442640305, 0.7247701287269592, 0.42556729912757874, 1.306161880493164, 0.4382716417312622, 0.7601836323738098, -0.4527682960033417, 0.4996058940887451, -0.16411471366882324, -0.9106036424636841, 2.2776577472686768, 0.5841041207313538, -0.14363881945610046, 0.6062076091766357, -0.3124888837337494, -1.3825675249099731, 0.09644877910614014, -0.7769331336021423, -1.9699420928955078, -1.8957890272140503, -0.6919710040092468, 1.7917383909225464, -0.12008623033761978, 0.28429195284843445, 0.10968739539384842, -0.081313356757164, 0.11091692000627518, 0.5929010510444641, -0.9539002776145935, 0.1677999645471573, 1.5118159055709839, -0.1809207648038864, -0.5135825872421265, 0.39729899168014526, 1.5212849378585815, -0.1557246893644333, 0.5787566900253296, 1.941909909248352, -0.4135408401489258, -0.4361550211906433, 0.36973440647125244, 0.6621730923652649, -0.4291629195213318, -1.2680567502975464, -0.8238141536712646, -1.4105713367462158, -0.37819617986679077, 1.381771206855774, 0.6316344738006592, 1.9235297441482544, 0.036178965121507645, 0.37663066387176514, -0.14410078525543213, 1.3558740615844727, 0.4427913725376129, 0.030472416430711746, 0.7885555624961853, -0.9838335514068604, -0.4334258437156677, 0.8263862729072571, 1.963757872581482, 0.6061744093894958, -0.6162803769111633, -1.4719600677490234, 0.9607974886894226, -1.6622440814971924, 0.28360703587532043, -0.46818894147872925, 0.609626054763794, 0.22947503626346588, 0.003641667077317834, -0.537860095500946, -0.21143409609794617, 0.6192454695701599, 0.015376308932900429, -0.37162041664123535, 0.4079238474369049, 1.2983847856521606, 0.17402024567127228, -0.48655185103416443, -1.4336947202682495, 0.2668587863445282, 2.633831739425659, 1.439326524734497, 0.054616644978523254, -0.3606084883213043, -1.2481369972229004, -0.03207455202937126, 0.6432427763938904, -1.1026655435562134, -1.7566957473754883, -0.8424146771430969, 0.15504160523414612, 1.6165697574615479, 0.014859674498438835, -1.376380443572998, -1.0610566139221191, -0.12035162001848221, 0.9167340993881226, 0.931033194065094, -0.26509398221969604, -1.0689213275909424, 0.8833817839622498, -0.13566970825195312, 2.118252754211426, -0.7549018263816833, -1.4293426275253296, -0.8527112007141113, -0.0906846895813942, -1.0105754137039185, -0.05298401415348053, -1.9533368349075317, -0.9824260473251343, 0.3253650665283203, 1.0463519096374512, -0.05318225920200348, 1.6091984510421753, -2.0791170597076416, 1.1804218292236328, -0.8766559362411499, -0.13158676028251648, -0.1677333414554596, -1.3220378160476685, -0.6041972637176514, 0.37025320529937744, 0.2313663512468338, -0.3204548954963684, 1.4942166805267334, 0.08937463909387589, 0.9975605607032776, -2.168604612350464, 0.7719194293022156, -2.4912679195404053, -0.24562641978263855, 0.6048359274864197, -1.7884516716003418, -1.4511505365371704, 0.47350648045539856, 0.09136722981929779, 2.285637378692627, -0.15953552722930908, 0.7321378588676453, -0.027414372190833092, 0.09588848054409027, -0.7729006409645081, 0.35637587308883667, -0.5503348708152771, 0.3579798936843872, 1.3823351860046387, 1.0800491571426392, 0.1680333912372589, 1.669156789779663, -0.4452788531780243, -1.7582343816757202, -0.05502944439649582, -0.8230708241462708, -0.21862098574638367, -0.8855583071708679, 0.4443778395652771, 0.08651707321405411, -0.7041285037994385, 0.6549718379974365, -1.0144238471984863, -1.6748666763305664, 0.1672399491071701, 0.06537561863660812, 1.2821543216705322, -0.24724499881267548, -0.6738829016685486, -1.2712241411209106, 0.29256710410118103, -0.5953843593597412, 0.8180710077285767, 0.12100067734718323, -1.856753945350647, 1.388422966003418, -0.7987666130065918, -0.20478160679340363, 0.434739351272583, 0.9573978185653687, -1.0995123386383057, 0.2861960828304291, 1.0716863870620728, 0.7124553322792053, -1.1464500427246094, 0.7294794321060181, 1.2573447227478027, 0.4278951585292816, 0.3679838180541992, -0.13149449229240417, -0.742262601852417, 0.28733694553375244, 0.4005601406097412, -1.9826335906982422, -1.4541364908218384, -0.550054132938385, 0.3453327715396881, 0.1858273148536682, -0.3575030267238617, -0.09133921563625336, 1.507212519645691, 0.8365280628204346, 2.256169080734253, 0.6103655099868774, -0.7273336052894592, -0.9805139899253845, -0.3593643307685852, 0.5698534250259399, 0.043522145599126816, 0.07770402729511261, -0.8946270942687988, 0.43638598918914795, 0.4471389949321747, -1.0597565174102783, -0.39062780141830444, 0.7346830368041992, -0.38630372285842896, -1.8305318355560303, 0.4555037319660187, 0.6756338477134705, 0.9496243000030518, -0.7867826223373413, 0.6322857141494751, -0.781994104385376, 0.1342555731534958, -1.3835142850875854, -0.45637503266334534, 0.5066288113594055, -1.3127161264419556, -1.4048535823822021, 1.6872304677963257, 1.2394142150878906, 0.7642408013343811, 0.1992017924785614, 0.16362214088439941, 0.8715762495994568, 1.4333237409591675, -1.0656583309173584, 0.21072623133659363, -0.6465228199958801, 0.4987674653530121, -0.867289662361145, 0.879804790019989, 0.8202904462814331, -1.05995512008667, -0.7054542303085327, -0.48015180230140686, 0.36355552077293396, -0.036008864641189575, -0.058363959193229675, -1.5063878297805786, 0.25286778807640076, -0.6325199007987976, 0.6406214237213135, 0.744476854801178, 1.3816312551498413, -0.4678073227405548, -1.1597809791564941, -0.24299709498882294, 0.014983118511736393, 3.084959030151367, 0.8612287044525146, -1.4422820806503296, -0.5286043286323547, -0.9497738480567932, 0.21978364884853363, 0.6718680262565613, 0.38069283962249756, 1.2506515979766846, 0.06713588535785675, -0.48678308725357056, 0.7512416243553162, -0.5336523652076721, -1.5404298305511475, -1.2868436574935913, 0.7270988821983337, 0.8053565621376038, -0.7989058494567871, 0.17105907201766968, -0.3684355318546295, 0.6487675905227661, -0.293696790933609, 0.26519402861595154, -0.5470128059387207, 0.24207675457000732, -0.8237699270248413, 1.1742548942565918, -1.6603977680206299, -0.7209959626197815, -0.3662063181400299, 0.055454809218645096, 0.5169509053230286, -2.2540440559387207, -0.11794096231460571, 0.45121729373931885, -1.861311674118042, -1.7040016651153564, 1.3908015489578247, 0.2701047360897064, 1.1645258665084839, 0.39921632409095764, -2.119244337081909, -1.5433712005615234, 0.8398507237434387, -0.03108074516057968, 0.15428821742534637, 2.1390037536621094, 0.3618362545967102, 0.3675174117088318, 0.6588774919509888, -0.5561067461967468, -0.08046414703130722, -1.4543808698654175, -0.2337997555732727, -0.984982430934906, 0.644353985786438, -0.8843521475791931, 1.545212745666504, 0.012444929219782352, -0.2986011505126953, 1.5759575366973877, -2.0040242671966553, 0.20555752515792847, -0.12844842672348022, 0.2867109775543213, -0.009208263829350471, 0.4701891243457794, 1.6151992082595825, -0.47969135642051697, -0.4586848020553589, 1.6684857606887817, -0.9973450899124146, 0.22864653170108795, 1.353804349899292, -1.2339409589767456, -1.0308160781860352, -0.040816761553287506, -2.3907816410064697, -2.643083095550537, 0.6007810235023499, -0.8492082953453064, 1.389914870262146, -0.12701979279518127, 0.4848022162914276, 0.7195899486541748, -0.12658479809761047, -0.10636401921510696, 1.4062594175338745, -0.9965128302574158, -0.8497040867805481, -2.076533317565918, 0.8384547233581543, -1.0315645933151245, 0.43539509177207947, -2.6602368354797363, -0.19349688291549683, 0.8073592782020569, 0.9279397130012512, -0.7298696637153625, 0.6804933547973633, -0.13007250428199768, -1.4622098207473755, -2.0725507736206055, 0.8632702827453613, -0.10802067071199417, 0.653671145439148, 0.6010223627090454, -1.510733962059021, 0.8098078370094299, -0.0008908018935471773, -0.6802945137023926, 0.6570606231689453, 0.20776653289794922, -0.23150505125522614, 0.096169613301754, -0.18739761412143707, 0.3502075970172882, -1.2311933040618896, 0.2593221664428711, 1.398034930229187, 0.5874426960945129, -0.7625380754470825, -0.4759898781776428, -0.7069662809371948, 1.3798913955688477, -0.06079438701272011, -1.205033302307129, 2.5968220233917236, 0.22085870802402496, 1.1840016841888428, 0.6940129995346069, 0.3558885157108307, 0.6554188132286072, 0.09991476684808731, 0.24507784843444824, 1.5989383459091187, 0.45950162410736084, 0.4718210995197296, -1.5775171518325806, 1.4762002229690552, 0.11714036762714386, -1.33732271194458, -0.0013958030613139272, -0.8774662017822266, -2.3974974155426025, 0.4433367848396301, -1.5531271696090698, 0.19648954272270203, 0.2790866792201996, -1.0856711864471436, -0.3808736801147461, 0.008703543804585934, -0.8993544578552246, -0.37341129779815674, -0.567051351070404, 0.945864200592041, 0.30989402532577515, 0.6150325536727905, 1.9058531522750854, -0.31229856610298157, 1.2342453002929688, -0.4504312574863434, -0.6493552327156067, -0.15611468255519867, 1.2849452495574951, 1.5625653266906738, 0.8646801114082336, 1.4511407613754272, 0.4784718453884125, -0.288565069437027, 1.5724691152572632, -0.26763197779655457, -0.7409438490867615, 1.0507597923278809, 1.5766713619232178, 0.8323221802711487, 0.26398587226867676, 1.1031044721603394, 0.1372852474451065, -0.48766598105430603, 0.42250338196754456, 1.074856162071228, 0.2262718677520752, -1.1894422769546509, 0.20344871282577515, -1.2278045415878296, -0.423349529504776, -0.5956292748451233, -0.5641298294067383, -1.063905119895935, -0.24468035995960236, 0.8743355870246887, -0.5002710819244385, 0.7654083967208862, 1.9485782384872437, 0.21886995434761047, -0.012796053662896156, 0.9489441514015198, 2.4059431552886963, 0.1896457076072693, -0.11194684356451035, 0.15791162848472595, -0.3470868766307831, -0.5429512858390808, 0.32233694195747375, -0.9732252955436707, -1.0023081302642822, 1.1454172134399414, 0.6739192605018616, -1.3475732803344727, 0.3027980625629425, 1.5303635597229004, -2.0763063430786133, -0.6740656495094299, -0.43277251720428467, -0.05635153129696846, 0.8307424187660217, -0.3234240710735321, -0.49563077092170715, -0.42939332127571106, 0.4058411717414856, 1.0508939027786255, 1.847955584526062, 1.1624367237091064, -0.9816344976425171, -0.2792244255542755, -0.0880807638168335, 1.6684761047363281, -0.14146722853183746, -0.8684067130088806, 0.0934920683503151, -1.14069664478302, 0.36116379499435425, -0.3019658625125885, -0.5543555617332458, 1.1724023818969727, 1.0362390279769897], [0.4155420660972595, 1.143869161605835, 0.35836145281791687, 0.6895617842674255, -2.6661739349365234, 0.5769228935241699, 0.2642367482185364, -0.06913889199495316, 1.3441294431686401, -0.11439739167690277, -0.6290290951728821, -0.01204601302742958, -0.06586398184299469, 0.19857077300548553, -1.4740819931030273, 1.3813450336456299, -0.614956259727478, -1.009813904762268, -0.6095330715179443, -1.6135188341140747, -0.5338627696037292, 0.4740860164165497, -0.4123438894748688, 0.1491885632276535, 1.3415145874023438, -0.8351465463638306, -0.6556687951087952, -0.049717117100954056, 1.7298709154129028, 0.9623685479164124, -1.9129414558410645, 0.6930224895477295, -0.4405183494091034, 0.7624726295471191, -0.4299117922782898, 0.2506227493286133, 1.4490526914596558, -1.8781429529190063, 0.9657837152481079, -1.7381377220153809, -0.10591644048690796, 0.3373294770717621, -2.175110101699829, 1.376800298690796, 0.8449037671089172, 0.6095109581947327, 1.3024600744247437, -1.1163582801818848, -1.4802234172821045, 0.25236207246780396, 0.00033473485382273793, -0.9865404963493347, -0.6403099894523621, -1.2499159574508667, 0.652776837348938, 1.0704267024993896, 0.6535403728485107, 1.2830413579940796, 0.49197205901145935, -2.2147605419158936, 0.43081384897232056, 0.3575650155544281, -1.226482629776001, 0.3636113405227661, 0.05356026813387871, 0.3799584209918976, 1.398208498954773, -0.36365649104118347, 1.3425966501235962, -0.2307247817516327, -0.4032585620880127, 0.44969284534454346, -1.6256107091903687, 0.561086118221283, -0.2588241696357727, 0.5612522959709167, -2.6650454998016357, -0.942442774772644, -0.2763049006462097, -1.4315791130065918, -0.5676606297492981, 1.1810568571090698, 0.10369329899549484, 1.5291342735290527, -0.48840776085853577, 0.2942267060279846, -0.05099623277783394, -0.6890040040016174, -0.7884931564331055, -0.7802268862724304, 1.2122166156768799, 1.0755622386932373, -0.9141572713851929, 0.5548532009124756, 0.2614974081516266, -0.8748800754547119, -2.0015859603881836, 1.4524319171905518, -0.10801532864570618, -0.2843778431415558, 0.10331310331821442, 0.26134437322616577, 0.9918683767318726, -1.1263655424118042, 0.9009286761283875, 0.13194146752357483, 1.1428163051605225, -0.8628366589546204, 0.30606845021247864, 0.5623704195022583, 0.49420657753944397, 0.3810681700706482, 0.6508519649505615, 0.48770907521247864, 1.4020273685455322, 0.34373602271080017, -0.25642526149749756, 0.9605962634086609, -1.0403802394866943, -2.291130781173706, 0.4046034812927246, 0.35675618052482605, 1.265256404876709, 1.1266430616378784, 0.17235596477985382, 0.5784317851066589, -0.8788979649543762, -0.10226095467805862, 0.9695432186126709, -0.3681446313858032, 1.7613122463226318, 0.25238490104675293, -0.4740297794342041, -0.3084114193916321, 2.7905516624450684, -0.42375844717025757, 0.06010245159268379, 1.5851552486419678, 0.4028452932834625, -0.41267576813697815, 0.8692013621330261, 1.8748304843902588, -1.3922593593597412, 1.468468189239502, 0.4688192307949066, -0.869869589805603, -1.7980029582977295, 1.0796290636062622, 0.2906333804130554, 0.4454021453857422, -0.18504981696605682, -1.4107223749160767, 0.25535261631011963, 0.5220015048980713, -0.5716593265533447, 0.6097570657730103, -2.2856454849243164, -0.084404356777668, -0.10628291964530945, -0.8802592158317566, -0.41439253091812134, 0.5351642370223999, -1.3030420541763306, -0.5044488906860352, 0.32158568501472473, -0.7133107781410217, -0.8328273892402649, 0.8682869076728821, 0.5504347681999207, 0.012382136657834053, -1.7840640544891357, -0.2789390981197357, -0.29233303666114807, -1.5111409425735474, 1.2451237440109253, 0.6327003836631775, 1.27450430393219, 0.03205690160393715, 1.0295662879943848, 1.9582635164260864, -1.0992265939712524, -1.7240475416183472, 0.08733049035072327, -0.8718160390853882, 1.3140747547149658, -0.23746956884860992, 0.3550586700439453, -2.279390811920166, 0.5917434096336365, -0.8910050988197327, -1.1666715145111084, 1.1757993698120117, 0.5237311124801636, 1.3142685890197754, 0.7844211459159851, 0.8792405128479004, 1.717254877090454, -1.136393666267395, -0.4095775783061981, 0.9737988114356995, 0.8165552616119385, -1.4882800579071045, 0.9072878956794739, 0.5274533033370972, 0.2134949117898941, 0.7608869075775146, -0.45469868183135986, -0.10996249318122864, -0.7206949591636658, -0.6415273547172546, -0.7289171814918518, 0.575558602809906, -0.7571138739585876, 0.12874236702919006, -0.6781931519508362, 0.09829825162887573, 1.9532198905944824, -0.27118101716041565, -1.5244624614715576, 0.008435579016804695, -1.2234841585159302, 0.8561068177223206, -1.036777138710022, -1.4650830030441284, -0.9590659141540527, -1.0966476202011108, -0.7434778213500977, 1.675638198852539, 0.6631640195846558, -0.6915954351425171, 0.044538483023643494, -1.2233067750930786, -1.1374757289886475, -0.560364305973053, -0.5317553877830505, -0.2203635424375534, 1.0156861543655396, 0.4181760251522064, -1.4088164567947388, 0.17555931210517883, -1.4811488389968872, -2.235678195953369, 1.0964417457580566, -0.3006507456302643, 0.0904541164636612, -0.7306416034698486, -0.5766714215278625, -0.410831093788147, -1.0408670902252197, 0.2849181294441223, 0.7094181180000305, 0.9376795291900635, -1.0143462419509888, 1.0148526430130005, -0.9050289392471313, -0.934634804725647, -0.44219037890434265, 0.6665821671485901, 1.1973514556884766, 1.9668482542037964, -0.01756405644118786, 0.33696985244750977, -0.5584367513656616, -1.5695515871047974, 1.0009568929672241, -1.8983228206634521, -0.3265339732170105, 0.6714772582054138, -1.518578290939331, -0.18875835835933685, -0.12597280740737915, -1.0778917074203491, -0.3440835773944855, -0.33272719383239746, 0.031232398003339767, -0.6910218596458435, 0.9319824576377869, 0.08470164239406586, 1.0504344701766968, 1.567170262336731, -1.0651730298995972, 0.8377667665481567, 1.9122116565704346, 0.3439648151397705, 0.45633843541145325, -0.23342855274677277, -0.29909148812294006, -0.4925846755504608, 2.3020622730255127, 0.436987042427063, -0.8830536007881165, 0.42474299669265747, 0.8941351771354675, -2.1378960609436035, 0.061710286885499954, -1.201560139656067, -0.6208842992782593, 0.9393268823623657, -1.046838402748108, 1.5868805646896362, 0.7167310118675232, -0.22827152907848358, -0.2655462920665741, 0.17334602773189545, -1.1554031372070312, -1.0401456356048584, -0.8618047833442688, -0.5945574641227722, -0.7138959169387817, 1.0479267835617065, -0.46192553639411926, 0.41441088914871216, 0.4054919183254242, 0.7847646474838257, -1.8038233518600464, 0.24653568863868713, -1.260043740272522, -1.315862774848938, -0.41924235224723816, 0.2569800317287445, -0.29263296723365784, 1.8614981174468994, -0.06600020825862885, 1.074161410331726, -0.05994429811835289, -0.5341621041297913, -0.3009832799434662, -0.2874162495136261, 0.8239866495132446, 0.5651190280914307, -0.4559594690799713, 0.36108794808387756, 1.7409167289733887, 1.4581197500228882, -1.67339289188385, -0.8905540108680725, -0.38395795226097107, 0.1592225581407547, 0.7483965158462524, 0.29879042506217957, -0.7384351491928101, -0.18367311358451843, -0.3595666289329529, 3.963500499725342, 1.2639437913894653, -0.37949687242507935, -1.693734049797058, 0.41139522194862366, 1.395264983177185, 1.1579331159591675, -1.5631948709487915, 0.3053928315639496, 0.6838766932487488, -1.4091452360153198, -0.40621083974838257, 0.5576145648956299, -1.2281171083450317, -0.9861982464790344, 0.5935878753662109, 0.025550970807671547, 0.22157183289527893, -0.03435000404715538, -0.24667367339134216, 0.1054731160402298, 0.004804960917681456, 0.25257790088653564, -1.0362117290496826, 0.7415156364440918, -1.269567608833313, -1.2372143268585205, -0.7203935980796814, -1.6159746646881104, 1.197270154953003, 0.27461257576942444, -0.9581833481788635, 0.2032402753829956, 0.40286684036254883, 1.016751766204834, -0.5495882630348206, -0.09431222081184387, -0.25399935245513916, 1.3078367710113525, -0.3525700271129608, -0.3313678205013275, 0.20480427145957947, -0.562480628490448, -1.4851527214050293, -1.0552308559417725, 1.252326250076294, -1.6621372699737549, -1.3327981233596802, 0.19138024747371674, -0.8032971620559692, 0.8868966698646545, 0.2082667499780655, -0.0746130719780922, -0.15366873145103455, -0.3146250247955322, -0.4935455918312073, -0.24775679409503937, 1.0735175609588623, 0.5801284909248352, 1.4369401931762695, -0.931219220161438, -0.4834365248680115, -1.4468648433685303, 0.8962645530700684, -0.8333267569541931, -0.3969511091709137, 0.38705018162727356, -0.19272083044052124, 1.7580512762069702, 0.2827438712120056, -1.7852147817611694, -0.3998611867427826, -1.9460011720657349, -1.7503914833068848, 1.534571647644043, 0.755346417427063, 0.6137537956237793, 2.7478859424591064, 0.691211462020874, 0.24452240765094757, -0.9290217161178589, 0.5072859525680542, 0.7743931412696838, -0.36318743228912354, -0.20283429324626923, 1.0772267580032349, -0.3556011915206909, -1.443917989730835, -0.1505875438451767, 0.40084707736968994, 0.3689451515674591, 0.20876751840114594, 0.8307504653930664, -1.2357184886932373, -0.49959638714790344, 0.2991226613521576, 0.6850447654724121, 0.25399473309516907, -1.0451370477676392, -0.6240193843841553, -0.16594228148460388, 0.05692587047815323, 0.8893314599990845, -1.3117127418518066, -0.33744609355926514, -0.9735140800476074, -0.6080670356750488, -0.21461474895477295, 0.0977206826210022, 0.8838424682617188, -0.8985376358032227, 0.5532494783401489, -0.3426932692527771, -0.11780655384063721, -0.9178382158279419, 1.4267877340316772, 1.1537086963653564, -1.3913424015045166, 1.011374592781067, 0.08820223063230515, -0.659532904624939, 0.744455099105835, -0.09871458262205124, -1.6714024543762207, -1.7698395252227783, -0.5974217653274536, 0.7385450005531311, 0.9836167693138123, 0.16577720642089844, 0.03754672408103943, -0.28593409061431885, -1.6196101903915405, 0.7160249948501587, -0.8679270148277283, 0.5880367159843445, -0.14451275765895844, 0.8705828785896301, -2.0478715896606445, -0.3458646237850189, 0.2985166311264038, -1.2995917797088623, -0.2849741280078888, -0.22417737543582916, -0.9693838953971863, 1.6536760330200195, 0.4870866537094116, -0.9265780448913574, 1.8085904121398926, 1.0821382999420166, 0.1680520623922348, -0.6523824334144592, 0.7495402693748474, 0.5986540913581848, 0.3253357708454132, 0.26578933000564575, 2.9962270259857178, -0.40196695923805237, -1.8473001718521118, 0.292809396982193, 0.13566456735134125, -0.8523099422454834, 2.4245340824127197, -2.4362614154815674, 0.6501518487930298, -0.8196870684623718, -0.356477290391922, 0.38564640283584595, -0.5644657015800476, 1.568317174911499, 0.08146876841783524, 0.24341420829296112, -0.8092722296714783, -0.514804482460022, -0.6409372091293335, -0.1075252965092659, -1.2125738859176636, 0.13196520507335663, 0.20036429166793823, -0.4070776700973511, -0.6234363317489624, 0.06350089609622955, 1.0572837591171265, -0.4195851683616638, 0.10255508124828339, -0.8963448405265808, 1.4088705778121948, -1.8231405019760132, 0.8476263284683228, -0.9325034618377686, -0.9758355021476746, -0.31452417373657227, -0.9826818704605103, 0.8355036973953247, -0.2972749173641205, -0.37924811244010925, -1.9109381437301636, -0.7598581314086914, -1.360346794128418, -1.6634873151779175, -1.5875781774520874, -1.3349196910858154, -1.077183723449707, -1.164363980293274, -0.0885358676314354, -0.31670695543289185, 0.13426640629768372, 0.4653834402561188, 1.073617935180664, 0.13697770237922668, 1.4483823776245117, -0.4516948461532593, 2.5673105716705322, -0.7267321944236755, -1.2602320909500122, 1.673168420791626, 0.3309824764728546, -1.0465842485427856, -1.650266170501709, 1.0381344556808472, -0.06274160742759705, 0.9063011407852173, -0.9446633458137512, -0.6020035147666931, -1.1064045429229736, 0.42156875133514404, -2.4132590293884277, -0.05440492182970047, 1.2727715969085693, -0.801505446434021, 0.16001451015472412, 0.24084100127220154, -0.5318131446838379, 0.4566418528556824, -0.03714008629322052, 0.29337742924690247, 0.042369574308395386, -1.2101337909698486, -0.6339960694313049, -0.07698627561330795, 0.19943943619728088, 0.5706173181533813, -1.054313063621521, 0.21981118619441986, -1.6947120428085327, -1.1018480062484741, -0.24379433691501617, 1.4206633567810059, -1.553863525390625, -0.3561847507953644, 0.025235293433070183, -1.6296577453613281, -1.647703766822815, -0.7442768812179565, -0.20183658599853516, 0.16937069594860077, 0.4864729642868042, -1.1207829713821411, -0.1506795585155487, 1.3116382360458374, -1.6786575317382812, -1.7518742084503174, -0.07222456485033035, -0.003448652336373925, 1.2301689386367798, -0.17977456748485565, -0.2766040861606598, 0.3621821999549866, 0.1479213535785675, -2.165621280670166, 0.18405982851982117, -0.28572750091552734, -0.001807592692784965, -1.3230986595153809, 0.1345321536064148, -0.6051914095878601, -1.202648401260376, -1.1032297611236572, 0.09389881044626236, 0.8810611963272095, 0.14534026384353638, -0.6517971754074097, 0.35504215955734253, 1.7363970279693604, -0.9975239634513855, -0.38102591037750244, 0.36643722653388977, -1.87236750125885, 0.3114360272884369, -0.49842822551727295, 0.9879190921783447, -0.8656339645385742, 0.8398237824440002, 1.0373424291610718, -0.8979597091674805, -1.0635420083999634, 1.025762915611267, 0.008519265800714493, 0.7421125173568726, 1.934070348739624, -0.15017065405845642, 0.12831588089466095, 0.0873531922698021, 0.288784384727478, 0.8165075778961182, -0.051901668310165405, 0.4158105254173279, 0.9789015054702759, 1.1662819385528564, -0.510333776473999, -1.3558599948883057, -0.9750293493270874, -0.45202207565307617, 0.6571505069732666, -0.2966958284378052, -0.7155410647392273, 1.6256786584854126, -0.7595824599266052, 0.5021376013755798, 1.1038135290145874, 0.21886001527309418, 0.6986731886863708, 0.017934562638401985, -1.1823740005493164, -0.5715748071670532, -0.774063766002655, -0.987949788570404, -0.43086400628089905, -1.4767221212387085, 1.2001827955245972, 0.1153082400560379, 0.9031169414520264, 0.9173939824104309, -0.5696669220924377, -0.23713763058185577, 0.16828827559947968, -0.315509557723999, -0.9509307742118835, -0.9996501207351685, -1.4309109449386597, -1.043667197227478, -0.02101108990609646, -0.41695818305015564, 0.728908121585846, -1.596449375152588, -0.8785861730575562, -0.1006382405757904, -0.06954867392778397, 0.6430930495262146, 0.5307096242904663, -0.07379329949617386, -0.37372007966041565, 0.9241127371788025, 0.9097522497177124, 0.9148178696632385, -0.17602384090423584, -0.7485455870628357, 1.0790612697601318, -0.3487405478954315, -2.2375640869140625, -1.4467837810516357, -1.0893888473510742, -0.3854377865791321, 0.16354423761367798, 0.01915678009390831, -0.7394793629646301, -0.2319256067276001, 0.8694374561309814, 0.6632249355316162, 0.32651960849761963, 2.788358449935913, -2.2531752586364746, -0.3067692816257477, -0.17506054043769836, -2.0262057781219482, 0.9208396077156067, -1.475227952003479, -1.792441725730896, -0.49069610238075256, -0.2576993703842163, -0.016459284350275993, -0.3234136998653412, -1.8508340120315552, -0.13775035738945007, 0.06009684503078461, 0.5516674518585205, 0.7334510684013367, -1.0615124702453613, -0.7756041288375854, -2.0026063919067383, -0.5414840579032898, 0.4658922553062439, 0.45295634865760803, -0.7774202823638916, -0.2600785493850708, 1.558885931968689, -0.2519925832748413, -0.22847677767276764, -0.6439781785011292, -0.1994507610797882, -1.370862364768982, 0.20069903135299683, 0.2941170632839203, -2.731079339981079, -1.813578724861145, -0.4689212441444397, -1.5778809785842896, -1.671730637550354, -0.6860983967781067, 0.2818828821182251, 0.14400725066661835, -0.6298801898956299, -1.40824556350708, 0.7240873575210571, 0.8424113392829895, 0.07241655141115189, 0.638735294342041, 0.5069689750671387, -1.2449893951416016, 1.0179001092910767, 1.2176460027694702, 0.4703901410102844, -0.14431032538414001, -0.5786875486373901, 1.8541842699050903, -0.31047308444976807, -0.20393887162208557, -0.39638790488243103, 0.17081201076507568, -0.4352407455444336, -0.9327963590621948, 0.02661905437707901, 0.8156548142433167, 0.09977070987224579, -0.6318567395210266, -2.2360126972198486, 0.3497058153152466, 0.9586168527603149, 1.260912537574768, 0.3136442005634308, -0.3696361184120178, -1.1883140802383423, -0.4493027925491333, -0.24678026139736176, -0.6768315434455872, 0.33577078580856323, 0.0021027871407568455, 1.5327270030975342, -0.8720602989196777, 0.33301231265068054, -0.34150105714797974, 1.5273464918136597, -0.8262255787849426, 0.4378264844417572, 1.2701377868652344, -0.692996084690094, -0.8766491413116455, -1.2549970149993896, -0.4904569685459137, 0.36798787117004395, -0.5397958159446716, 0.14079967141151428, 0.6426451206207275, -0.2511730194091797, 1.200822114944458, -0.2983189523220062, -0.5867172479629517, 1.0842068195343018, -0.4495043456554413, -0.37909364700317383, -0.45920678973197937, 0.7154861688613892, 0.5575457811355591, 0.23708777129650116, 1.4558554887771606, 0.45685631036758423, 1.4133416414260864, -0.32217279076576233, -0.5059062838554382, 0.28268900513648987, -0.5344276428222656, -0.8821864128112793, -0.6759735345840454, 0.01725155860185623, -0.5332647562026978, -0.4406454265117645, 0.358412504196167, -0.6955065131187439, -1.2325679063796997, 0.16034995019435883, -0.24323737621307373, -1.5782097578048706, 1.3403204679489136, 0.26644647121429443, -0.12134698778390884, 1.5252869129180908, 0.7160158753395081, 0.729324221611023, -0.947225034236908, -0.6445754170417786, -0.927733838558197, -0.8194834589958191, 0.6808260083198547, 0.9752237200737, -1.9011085033416748, 0.008820760063827038, -1.1799969673156738, 0.025624243542551994, 0.4172378182411194, -1.2630037069320679, -1.4029101133346558, 0.4436119496822357, 0.5546625256538391, 2.543792724609375, 0.5661417245864868, 1.1619030237197876, 0.9967118501663208, -0.44031253457069397, -0.39520666003227234, 0.1156015545129776, 1.1448549032211304, 1.0742863416671753, -0.4417873024940491, -2.350554943084717, -0.7998053431510925, -0.7098885774612427, -0.8059759140014648, 0.7948170900344849, -0.8659040331840515, 2.0090363025665283, -1.4573752880096436, 0.10312855988740921, 0.8547126650810242, -0.6576905846595764, -0.14326098561286926, -1.295433521270752, 0.993037760257721, -0.6058069467544556, 0.8983375430107117, -1.5571593046188354, 0.8964990973472595, 0.08824276924133301, 0.5931021571159363, 1.0222986936569214, -0.5103861689567566, -1.0161948204040527, 0.930237352848053, -0.3040519952774048, -1.3954581022262573, 1.2722035646438599, -1.3062304258346558, -0.0733848288655281, 1.9973009824752808, 0.5088046193122864, 0.41397029161453247, 1.8167933225631714, 1.0358582735061646, -0.8371961712837219, 0.5390536189079285, -0.24367620050907135, -1.172332525253296, -0.7739466428756714, 1.9747143983840942, -0.3294049799442291, -2.851306915283203, 0.4939225912094116, 1.4012587070465088, 0.8865746259689331, -0.7161030769348145, 0.5994447469711304, -2.0698020458221436, -1.0678563117980957, -0.9220152497291565, 0.4086899161338806, -0.7836897969245911, 0.23575136065483093, 1.206895112991333, -0.43924185633659363, 0.047640420496463776, -0.7907015085220337, -1.6646276712417603, -1.0527395009994507, -0.27165141701698303, -0.30572736263275146, -0.06759072840213776, 1.2111353874206543, 0.526873767375946, -0.09230714291334152, -1.774693489074707, -0.007678554859012365, -0.8012490272521973, -1.481856107711792, -0.045963216572999954, 0.3076489269733429, 1.0144338607788086, -1.1871447563171387, -0.7309494018554688, 0.036722201853990555, -0.6354893445968628, 0.7413501143455505, 0.1713477224111557, 0.5663635730743408, -1.5165126323699951, -0.23313124477863312, -0.4438649117946625, 0.179897740483284, 1.6177890300750732, -1.2933748960494995, 0.32043734192848206, 1.059598684310913, -1.1916236877441406, -1.2094621658325195, 0.297026664018631, 1.6280512809753418, 0.1580793410539627, -1.0885205268859863, 0.08718516677618027, -0.9510930776596069, 0.1810021549463272, -0.5242841243743896, 0.90151447057724, -0.0034809268545359373, -0.31055396795272827, 0.4967043697834015, 0.06698804348707199, 1.8523056507110596, -0.17951549589633942, -0.37249159812927246, 0.06136387214064598, -1.8640345335006714, 0.4171580374240875, -0.6187644004821777, -2.1097848415374756, 0.008945602923631668, 1.4903205633163452, 2.054626226425171, 0.7607472538948059, 1.224004864692688, 0.588062047958374, -0.7415658235549927, -0.628261148929596, -2.0542209148406982, -0.9627494215965271, -0.7767238616943359, -0.9784618616104126, 0.5593284964561462, 0.5326736569404602, -0.9763243198394775, -0.9811911582946777, -0.059182994067668915, 1.8672469854354858, 0.20209456980228424, -1.9532122611999512, 0.24388502538204193, -1.0288265943527222, 0.9518274068832397, -0.4384003281593323, 0.9014890789985657, -0.0018846620805561543, 0.4802339971065521, -2.32214617729187, 1.7459384202957153, -1.0633759498596191, 0.19216671586036682, -0.2160424292087555, -0.5670555233955383, 0.5449764728546143, -0.42092931270599365, -0.9208667278289795, 0.6360798478126526, 1.6832839250564575, -0.5307386517524719, -1.702748417854309, 0.6179091334342957, 0.09068683534860611, -1.8362470865249634, -1.0690186023712158, -0.2033475935459137, -1.8048287630081177, -2.060720443725586, 0.4625970721244812, -0.8239646553993225], [0.13172189891338348, 0.06345269083976746, 0.9237084984779358, 0.5318661332130432, 0.11762865632772446, -0.4170109033584595, -0.6747018694877625, -1.318739652633667, -0.1020924523472786, 0.5926734805107117, -2.128598928451538, -0.4543837904930115, 0.14068318903446198, -1.5412719249725342, -1.8328388929367065, 0.12900620698928833, 0.9554936289787292, 0.5408170223236084, 0.42053285241127014, -1.5275297164916992, 1.2814319133758545, -1.0684852600097656, -0.4316467046737671, -0.04171719774603844, 0.6618664860725403, 2.014936685562134, 2.087768316268921, -1.2269234657287598, 0.20340703427791595, 0.6348809003829956, 0.6804025173187256, 0.30835235118865967, -2.4513931274414062, 1.0461245775222778, 1.848504900932312, -0.49563834071159363, 0.8014801740646362, -0.29340243339538574, -2.0697150230407715, 0.42417111992836, -0.20633703470230103, 0.23659747838974, -0.12143582105636597, -0.3993438482284546, -0.5674533247947693, 0.9273466467857361, 1.3234087228775024, -1.2210348844528198, 0.2858659625053406, -0.4075133502483368, 0.1532236486673355, 0.28301921486854553, 0.25086304545402527, 1.6285269260406494, 1.0896639823913574, -2.0502376556396484, -0.6235026717185974, -1.106197714805603, 0.1761726438999176, 0.8597977757453918, -0.003170592710375786, -0.8886532187461853, -0.6750190854072571, -0.2365574985742569, 0.3803364336490631, 0.5648857951164246, 0.8545126914978027, -1.6164140701293945, -1.3599411249160767, 3.057194471359253, 0.6759960055351257, -0.31758174300193787, 0.12285876274108887, 0.5652890801429749, 0.27913954854011536, 0.7575799226760864, 0.18218117952346802, -0.31400197744369507, -0.2923722565174103, -0.26566192507743835, 0.2212907075881958, -0.5903189778327942, 0.6743787527084351, -1.7155725955963135, -0.6742426156997681, 2.318840265274048, 0.8279666900634766, 0.15520305931568146, 1.8577500581741333, 0.021416692063212395, 0.31120502948760986, 0.5042990446090698, -0.9213396906852722, -0.36828556656837463, -0.8614999055862427, -2.469423532485962, -0.6566327810287476, 1.0982407331466675, 0.44149649143218994, -0.0842244103550911, 0.6235464215278625, 2.0324926376342773, 0.3233194053173065, 0.2495976984500885, 0.8566279411315918, -0.004870936274528503, 0.15334293246269226, 0.4311303198337555, 1.4402199983596802, -0.8405135869979858, 0.5423064827919006, 1.7057173252105713, -1.4702696800231934, -0.17407040297985077, 0.48440077900886536, -1.1753102540969849, -0.5495108962059021, 0.3899000883102417, -1.159012794494629, -0.7404706478118896, 0.990020751953125, 1.4272913932800293, -1.4250503778457642, 1.3212093114852905, -0.3693733215332031, -0.5737354755401611, -0.3696490526199341, -0.7459492087364197, -0.43458226323127747, 0.7597606182098389, 1.927426815032959, 0.9820383191108704, -0.4990237057209015, -1.6579043865203857, -0.09962838888168335, -1.0322937965393066, 0.08877787739038467, -0.050785161554813385, -1.8200342655181885, 0.6986185908317566, 0.7535800337791443, 0.8983916640281677, 0.6876181960105896, -0.5635324120521545, -0.7927535176277161, -0.9310508966445923, -1.5036298036575317, 0.5497854351997375, 0.2657271921634674, -1.9599790573120117, 0.17079074680805206, 1.4390231370925903, 0.7064669728279114, -0.8574607372283936, 1.5547866821289062, -0.5423066020011902, -0.23212413489818573, -1.2514568567276, 0.49292683601379395, 1.665440559387207, -0.4153946340084076, 1.301805853843689, 0.5606045126914978, -0.2323707640171051, -1.057889461517334, 0.5054120421409607, -0.20722101628780365, -1.2838828563690186, 2.3748672008514404, -0.3370424807071686, 2.00412917137146, -1.0530669689178467, 0.3599646985530853, 0.1604916900396347, -1.1622631549835205, -2.1336653232574463, -0.4398070275783539, -0.11404705047607422, -0.0787830501794815, -0.7457053661346436, 0.5813599228858948, -0.7973873019218445, 2.3206217288970947, 0.33821094036102295, 1.4004696607589722, 0.3400084972381592, -0.0707448348402977, -0.07726918905973434, 1.9627121686935425, -0.2682397961616516, 0.6541242003440857, 1.1674344539642334, 0.44977322220802307, -0.7363070845603943, -0.42264723777770996, 1.297424077987671, 0.7219566106796265, 0.33968695998191833, 0.7628674507141113, 0.745636522769928, 0.5668342709541321, -0.5317953824996948, -0.019865693524479866, 1.02015221118927, 0.08052415400743484, 2.1820859909057617, -1.13104248046875, -0.13704854249954224, 0.5486133694648743, -0.19141970574855804, -1.0464555025100708, 1.4643921852111816, -0.43276217579841614, -1.430138111114502, -0.8805743455886841, 1.0496070384979248, 1.5616461038589478, -1.531693935394287, 1.2605352401733398, -0.0698942095041275, 0.17133747041225433, 0.016471846029162407, 0.7088645100593567, 0.49551767110824585, 0.5194200873374939, 0.7384666204452515, -0.2893311381340027, -0.07454892247915268, -1.0842289924621582, 0.8871446251869202, -1.1103121042251587, -0.571972668170929, 0.7647906541824341, -0.7683439254760742, 0.716740071773529, 0.6888576745986938, -1.7807135581970215, 0.05809350684285164, -0.05535317212343216, -0.19258838891983032, -0.6607134938240051, -1.3486882448196411, -1.184004783630371, -3.047619104385376, 0.19888131320476532, -0.5578658580780029, -1.7325036525726318, 0.6853606104850769, 0.011280127801001072, 0.512958824634552, 0.015196015127003193, -1.5060701370239258, -0.9330923557281494, -0.22553128004074097, -1.707136631011963, 0.5423542857170105, 0.36871710419654846, 0.24301829934120178, 0.7511871457099915, 1.6611312627792358, -0.7034299969673157, -0.3225522041320801, -1.175362229347229, -0.020454999059438705, -0.08008595556020737, -0.21088632941246033, -2.0366175174713135, 0.35626456141471863, -0.5394858121871948, 1.6906123161315918, -1.190189242362976, -0.3118113875389099, 0.875392735004425, 0.8335965871810913, 1.4341202974319458, -1.7253690958023071, -0.25394049286842346, 0.18084938824176788, 0.8140904903411865, -0.7829149961471558, 0.5548998713493347, 0.9691162109375, 0.5415919423103333, 1.474224328994751, 0.8706194758415222, -0.8440144062042236, -1.3646535873413086, -0.6584995985031128, 1.5801366567611694, -0.9193803668022156, -0.8094136714935303, -0.36053135991096497, -2.7817277908325195, -0.4340604245662689, -0.29619455337524414, 0.8863559365272522, 0.3271583616733551, 1.3001340627670288, 0.45706841349601746, -0.009914896450936794, 1.4877607822418213, -0.59151691198349, 0.19160063564777374, 0.36256712675094604, 0.31124961376190186, 0.8984929919242859, 0.6810408234596252, -3.207040309906006, 0.5745921730995178, 0.39088037610054016, -1.6302913427352905, -0.3840506672859192, -1.3946075439453125, -0.3495372235774994, -0.8041538000106812, -0.8913375735282898, -0.826831042766571, -1.9387173652648926, 0.750059962272644, 0.9668036103248596, 1.8254501819610596, -0.8037286400794983, -1.5720772743225098, -0.25753581523895264, 1.736368179321289, -1.6153723001480103, -0.8688282370567322, 0.698162853717804, -1.4727654457092285, -0.4364370107650757, -0.22250288724899292, -1.9064327478408813, -0.9344765543937683, -0.7017298936843872, 0.5056668519973755, 0.9456263780593872, 0.18460382521152496, -1.0273611545562744, -0.16112050414085388, -1.3310853242874146, 1.5114539861679077, -0.716981053352356, 2.7723803520202637, -0.3881209194660187, 0.20767083764076233, -2.270421266555786, 0.1083718091249466, 0.5189297199249268, -0.2206033319234848, 0.6673732399940491, 1.2665190696716309, -0.9389615058898926, -0.4750223159790039, -1.4663913249969482, 0.12585823237895966, -0.28079113364219666, -0.6832278370857239, -1.5880041122436523, -0.9596334099769592, -0.9603618383407593, 1.5837383270263672, -0.08485950529575348, -1.2338594198226929, 1.9281935691833496, -0.6159164309501648, 0.2700433135032654, 0.6951226592063904, -0.16017788648605347, -1.3086364269256592, 0.603326678276062, 1.6894323825836182, 0.32633838057518005, 0.8967887759208679, 0.4889695942401886, 0.020411023870110512, 0.13420557975769043, 0.8655359148979187, 0.43103381991386414, 0.8570534586906433, 0.17997325956821442, 0.47105109691619873, -0.8373410105705261, -0.2794937491416931, -0.8583709001541138, 0.19546031951904297, 0.568449854850769, -1.3633534908294678, 0.8556535243988037, 0.23708872497081757, 0.9292885661125183, -1.4197802543640137, 2.4109437465667725, 0.17167679965496063, -0.4574534296989441, 0.45062804222106934, 0.46517500281333923, -1.2122677564620972, 2.1794469356536865, -1.7291733026504517, -0.3313773274421692, 0.21959882974624634, -0.630291223526001, -0.8439096808433533, 1.3933756351470947, 0.04294730722904205, -0.439219206571579, 1.8342629671096802, 1.2313838005065918, 0.27187761664390564, 0.866466760635376, 0.1404881328344345, 0.355629026889801, 0.278842568397522, 1.0429949760437012, 0.19399911165237427, -0.6940819621086121, -0.8355883359909058, 0.4336916506290436, -0.6627665758132935, 0.05508028343319893, -0.3842999041080475, -2.0818493366241455, 0.873496413230896, 0.36988258361816406, 1.4085334539413452, -0.06899961084127426, 1.2217401266098022, -0.8498953580856323, 0.3123835027217865, 1.1822080612182617, 1.11745023727417, -0.39017805457115173, -0.9030836224555969, 1.7468500137329102, 0.19977742433547974, 0.5621254444122314, 0.674766480922699, -1.3034478425979614, 1.9741402864456177, -0.9712274670600891, -1.8580628633499146, 1.4102756977081299, -0.7355101108551025, 0.5730105638504028, -0.40744516253471375, 0.249247208237648, 0.5466596484184265, 1.406294584274292, -0.2817189693450928, -0.7785407304763794, -0.1745821237564087, -0.0019523241790011525, 0.2210070639848709, -0.29757413268089294, 1.7836902141571045, 1.919387698173523, -0.09320109337568283, -0.22953885793685913, 0.1646638661623001, -0.7379785776138306, 0.5510343313217163, 0.2612673342227936, 1.4074524641036987, -0.8559874892234802, 0.7221975922584534, -1.6245009899139404, -1.5799744129180908, 0.653620183467865, -2.7968389987945557, -1.2163187265396118, 0.9897392392158508, 0.6588466167449951, -1.9213647842407227, -1.5809390544891357, -1.9453134536743164, 0.7382019758224487, -1.6266226768493652, 0.28551703691482544, -1.1723089218139648, -0.5309354066848755, -2.62613582611084, 0.6665328741073608, 0.8460310101509094, 0.010899311862885952, 1.076442003250122, 0.188966304063797, 0.37951380014419556, -0.26691752672195435, -0.8907442092895508, 0.02950146794319153, 1.110892415046692, 0.915034830570221, -0.7855321168899536, 0.6417955756187439, 0.6935427188873291, 0.4027792513370514, -1.7176148891448975, 0.27707675099372864, -0.28912287950515747, -1.934537649154663, -0.1133643090724945, 0.6937353610992432, -0.43619194626808167, 0.30142316222190857, 1.1414519548416138, -1.3236591815948486, 0.6437495350837708, 1.2036055326461792, 0.5890001654624939, -1.7214970588684082, 1.1633983850479126, 1.765534520149231, 1.1373802423477173, 0.6051927804946899, -0.40455299615859985, -1.7510063648223877, -1.381303310394287, -1.399196743965149, -1.3708114624023438, -0.5837146043777466, 0.4813959300518036, 1.3560256958007812, -1.283815860748291, 2.14837908744812, 1.866259217262268, 0.22000467777252197, -0.3828863203525543, 0.23988118767738342, 0.8413373231887817, -0.9368067979812622, 1.0607496500015259, 1.4074835777282715, -0.07693514972925186, -0.2603954076766968, 0.548464298248291, -0.8341941833496094, -0.32848742604255676, -0.4227072298526764, -0.4724055528640747, 0.0737796425819397, 0.6418684720993042, -0.16074922680854797, 0.9233288764953613, -1.0484594106674194, 1.4117909669876099, -0.08584347367286682, 0.012111028656363487, -0.4496477246284485, -0.3417409658432007, 1.0116591453552246, -0.802479088306427, 1.3660192489624023, 0.992003858089447, 0.25173237919807434, 0.7772505283355713, 0.49293187260627747, 0.2998918890953064, -0.9012640118598938, 2.051920175552368, -0.9950206279754639, 0.11079162359237671, -1.0878677368164062, 1.028538703918457, -0.21942085027694702, 0.7551629543304443, 0.697546124458313, -0.7123335599899292, 0.6237406730651855, -2.343445301055908, 0.8677605986595154, -0.8846473097801208, 0.9961190223693848, 0.13434873521327972, -0.465558260679245, -0.021028975024819374, -1.6948399543762207, 1.7545413970947266, -0.8304568529129028, -0.014821468852460384, 1.769126296043396, 0.8831613659858704, 0.29645392298698425, -1.7306181192398071, 1.0862059593200684, -0.9813052415847778, 1.3383519649505615, -0.6744456887245178, 0.23018108308315277, 0.6363568902015686, 0.6935311555862427, 0.5857195258140564, 0.059863246977329254, 0.3575171232223511, -1.100495457649231, -1.805963158607483, 1.6921790838241577, -0.3017083406448364, 0.9178034663200378, -1.4154125452041626, 1.1485646963119507, 0.15310728549957275, 0.37882816791534424, 0.6187887787818909, 0.16679777204990387, -1.3203387260437012, 0.09853978455066681, -1.153306245803833, 1.498302936553955, 0.22922168672084808, -1.2925057411193848, -0.4905267655849457, -0.939414918422699, -0.6941607594490051, 0.2660154700279236, -1.656732201576233, -0.4418594539165497, -1.1689425706863403, 0.2968759834766388, 0.18458889424800873, 1.557289481163025, 0.0057054003700613976, -0.5740076303482056, -0.6457845568656921, -0.7649121284484863, -1.250931739807129, -0.8308383226394653, 0.13039523363113403, -1.0456892251968384, -1.583260178565979, 2.387136936187744, 0.6313458681106567, 1.0742316246032715, -1.7453343868255615, 1.1955291032791138, -0.6938310861587524, -1.0234427452087402, -0.1299152374267578, 0.41469240188598633, 0.16177774965763092, 0.1696174591779709, 1.88558828830719, 1.126476764678955, 0.5428289771080017, -0.6613056659698486, -0.4508746564388275, -1.2379858493804932, 0.9477291703224182, 0.7975229024887085, 2.1404879093170166, 0.3666416108608246, 0.410909503698349, -0.9696341753005981, 0.340948224067688, 0.3055395781993866, -0.37655240297317505, 0.07671476900577545, 0.608968198299408, 2.1040496826171875, -2.3469035625457764, 0.8002938628196716, -0.221856951713562, -0.8323450684547424, -0.3922106921672821, -2.6343889236450195, 1.2688521146774292, -1.0011481046676636, 1.8279768228530884, 1.2646228075027466, -1.3487865924835205, 0.15099333226680756, 0.6391493678092957, 0.8951168060302734, -0.5051537752151489, 1.0145281553268433, -0.23037296533584595, 1.7095850706100464, -1.2254983186721802, -1.053198218345642, -1.972373127937317, -0.11002861708402634, 1.0533438920974731, -1.2644404172897339, -0.11835163086652756, 1.022976279258728, -0.2975563704967499, 1.3562227487564087, 0.8091345429420471, 0.7656099796295166, 0.8296278715133667, 1.3999526500701904, 0.5824955105781555, -1.1202099323272705, -0.5978116989135742, -1.1445232629776, -0.6671742796897888, 0.17074312269687653, 0.46148481965065, 0.6509340405464172, -1.7850337028503418, -0.8828073143959045, -0.9078724384307861, -0.6458747982978821, 1.2245335578918457, -2.304992914199829, 1.1086887121200562, -0.10541144758462906, 1.4356639385223389, -1.6920818090438843, 0.6402522921562195, -0.929252564907074, 0.46256643533706665, 1.2238017320632935, -0.22774317860603333, -0.04889032989740372, -1.4485180377960205, -0.5684506893157959, 0.0983959510922432, -0.4604046642780304, -0.3173055946826935, -1.5906096696853638, -2.824773073196411, -0.03932524845004082, -0.07785651832818985, -1.249884843826294, 0.4639029800891876, 2.502775192260742, -0.6738575100898743, 0.4541094899177551, 0.9327946305274963, 0.9534591436386108, 0.7347478270530701, -1.2405791282653809, 0.06605468690395355, -1.7672971487045288, -0.38479334115982056, -1.5299417972564697, -0.4570244252681732, 2.1924219131469727, -0.5462633967399597, 0.6738807559013367, -0.27289092540740967, 0.6282843351364136, 0.8244251012802124, -0.49749550223350525, -0.20527739822864532, -0.255998820066452, -1.809550404548645, -1.3245760202407837, -0.6782605648040771, -1.82232666015625, 1.2462352514266968, -0.10132919251918793, 0.46736592054367065, 0.2009020894765854, 0.9241780638694763, -2.016890048980713, 0.4501321613788605, 0.6196826696395874, 0.38272523880004883, 0.2546977400779724, -1.0374740362167358, 0.891772449016571, -1.6300849914550781, -0.6151333451271057, -0.24575048685073853, 0.4385330080986023, 0.888247549533844, 1.535702109336853, 0.5762727856636047, -0.29283952713012695, 0.5874024629592896, 0.06689445674419403, -0.7321733236312866, 0.0016738346312195063, -0.3641449213027954, -0.12478026747703552, -1.2571204900741577, -0.35833224654197693, -1.2734209299087524, 2.0726847648620605, 0.14570192992687225, -0.07242472469806671, -0.7256821990013123, 1.979909062385559, -0.5182411074638367, -1.3725868463516235, 0.9548332095146179, -0.21949827671051025, -0.3645097017288208, 1.6287214756011963, -0.3457624614238739, 0.8697096705436707, -0.6022310853004456, -1.192699909210205, -0.8117355108261108, 0.13169996440410614, -0.24496817588806152, -1.1970492601394653, 1.353057861328125, -0.025318598374724388, -0.7855101823806763, 0.4198978543281555, -0.6114801168441772, -0.7782087326049805, 0.46943968534469604, -1.7949483394622803, -0.22096872329711914, 1.2895593643188477, 1.6169675588607788, -0.11436766386032104, 1.065801978111267, 0.638590395450592, -0.5077399611473083, -1.0554356575012207, -0.0041192020289599895, -0.9922480583190918, 1.3555749654769897, 0.04674253612756729, 3.253836154937744, 1.223079800605774, -0.05575226619839668, -0.3527761697769165, -1.6322295665740967, -1.0855813026428223, 1.0185372829437256, -0.28222736716270447, 0.5220146775245667, -1.185927391052246, -2.4003045558929443, -0.3122037649154663, -0.10361578315496445, -1.1913857460021973, -0.6882728338241577, -0.4072354733943939, -0.2661409378051758, -0.029090335592627525, -0.13258442282676697, 1.5479319095611572, -0.7007858753204346, -1.4406416416168213, 1.4647984504699707, 0.12849153578281403, -0.49721989035606384, -1.2987115383148193, 0.18555760383605957, 0.8804181814193726, 0.09870336949825287, 0.04996395856142044, -0.2049054652452469, -0.5306582450866699, 0.0824788436293602, -0.3166002333164215, 0.15322212874889374, -0.20288948714733124, 1.5766979455947876, -0.19811737537384033, 0.48474955558776855, 1.058149814605713, -0.6198581457138062, -0.4138718247413635, -2.3068885803222656, 1.014060378074646, 0.974549412727356, 0.9058607816696167, -0.5260202288627625, 0.18758325278759003, -1.7800073623657227, -0.11140752583742142, 0.33585843443870544, -0.5017859935760498, 1.5415759086608887, -0.7460325360298157, -0.6283866167068481, -0.91397625207901, 1.658735990524292, 0.7714288234710693, 0.8714495301246643, 0.02483493834733963, -1.8090626001358032, 0.11932100355625153, 1.044338345527649, 0.8852073550224304, 0.14607921242713928, -1.439321517944336, 1.8181350231170654, -1.273054599761963, -1.7375198602676392, -0.3228442966938019, 3.3943135738372803, 0.6788676977157593, -0.510208785533905, -2.524127244949341, -0.33950892090797424, -0.8729991316795349, 1.4476367235183716, -0.3938142657279968, 0.4717162549495697, 0.7053729891777039, -0.04356381297111511, 0.10181478410959244, 1.0479172468185425, 0.24491383135318756, -0.14699840545654297, 1.462595820426941, 1.8682914972305298, 0.8639596104621887, 1.1569044589996338, -0.8036573529243469, -0.05605762079358101, -0.9332197308540344, 0.13669021427631378, -0.2990237772464752, -0.5202422738075256, -0.41902580857276917, -1.5093744993209839, 0.009252937510609627, 1.0950478315353394, -1.356784462928772, -0.20756535232067108, -0.5166395306587219, 1.7702991962432861, -0.21253398060798645, -0.010914086364209652, 1.193935513496399, 0.9846099019050598, -1.2011224031448364, -0.13605566322803497, 0.3923933207988739, 0.7136147618293762, -1.5662122964859009, -1.8628058433532715, -0.48387041687965393, -0.8230971693992615, -0.1554068773984909, 0.9765260815620422, 1.0898338556289673, -0.7938675284385681, 0.6832165122032166, -0.14998145401477814, 0.16985562443733215, 1.5165375471115112, 0.6473600268363953, -0.8906063437461853, -0.5172825455665588, 0.5509187579154968, 1.8301665782928467, -0.2284960150718689, -0.3119469881057739, 0.5900008678436279, -0.4998197853565216, 0.182874858379364, -2.081702709197998, -0.13763394951820374, 1.2392246723175049, -0.8158066272735596, 1.3007820844650269, -0.8167305588722229, 0.96797776222229, 0.06357844918966293, 0.21844890713691711, -1.640784740447998, -1.0619629621505737, -0.21514299511909485, 1.6602329015731812, 1.4354146718978882, -0.7227468490600586, -0.6334906220436096, -1.2215265035629272, 1.0684794187545776, 1.4405012130737305, -2.2095251083374023, -2.020759105682373, -0.5009560585021973, -0.9065096378326416, 0.5296108722686768, -0.15119285881519318, -2.4428811073303223, 1.3133975267410278, 1.2617031335830688, 1.55846107006073, 0.05343281477689743, -0.058866530656814575, 0.8545692563056946, 0.6795307397842407, 0.31530848145484924, -1.4086322784423828, 0.5957130193710327, 0.46464917063713074, -0.8586384654045105, 0.22698451578617096, 1.0072015523910522, -0.35328546166419983, -1.2311710119247437, -0.7160623073577881, 0.20939317345619202, 0.6257573962211609, 1.8662073612213135, -0.012536394409835339, -0.06095059961080551, 0.25192320346832275, -0.8857642412185669, -1.599951148033142, 1.5380699634552002, -1.650956392288208, 1.0189764499664307, -1.6617375612258911, -0.08527391403913498, -0.6407699584960938, -0.20235516130924225, 0.5431380271911621, 1.3375492095947266, 0.0848277136683464, -0.12913331389427185, 1.2748926877975464, -1.2414299249649048, -1.519309401512146, 0.28569361567497253, 1.3968098163604736, 0.4221005141735077, -0.20039281249046326, -1.1950711011886597, 0.5100678205490112, -1.0318430662155151, -0.8994359970092773], [-0.08102558553218842, -1.682016372680664, -0.9308931827545166, 1.326653242111206, 0.5073115229606628, -0.23077449202537537, -0.11562894284725189, -1.1045676469802856, -0.08430324494838715, 0.1537574827671051, 0.3636108934879303, -1.8004533052444458, 0.3001464009284973, -0.241084486246109, -0.5286806225776672, -2.0811617374420166, -0.8777514100074768, 0.8952639698982239, -0.16801024973392487, -0.5806705355644226, -1.2913020849227905, 1.9062178134918213, -0.4143989384174347, -1.187172770500183, 1.0814878940582275, 0.09630413353443146, 0.5361364483833313, 1.4724913835525513, 0.451987624168396, -1.0511218309402466, 1.4194341897964478, 0.34860992431640625, -0.13925209641456604, 0.2523953914642334, 1.3911211490631104, 0.8115861415863037, -0.7893301248550415, -0.20744569599628448, -0.2273118644952774, 0.5062870979309082, -0.4861070513725281, -1.4726155996322632, 0.6330757737159729, 0.4334753453731537, 0.197550430893898, 1.035557746887207, 0.8548586964607239, 0.44917553663253784, -2.8243448734283447, -2.42042875289917, -0.47931623458862305, -1.3585587739944458, 1.4138907194137573, -1.209157109260559, 0.4786660671234131, -0.49815571308135986, -0.26925522089004517, -0.4322173297405243, -2.455758810043335, -0.44691041111946106, 2.075085163116455, -0.04747655242681503, -0.25554320216178894, 0.3534301221370697, 0.007660869508981705, -0.8941112756729126, -1.7048664093017578, -0.4747324287891388, 0.6410296559333801, -0.528982400894165, -0.025055911391973495, 0.07524700462818146, 0.6248000264167786, -0.18717245757579803, 0.4650481641292572, -0.3661006689071655, 0.6176813840866089, -0.124374620616436, 1.5668092966079712, 0.26732882857322693, -0.9558630585670471, -0.7821647524833679, 0.7514515519142151, -1.2549570798873901, 0.9569019675254822, -0.5491679310798645, -0.9332076907157898, 0.5912030339241028, -0.9925090670585632, -1.811867117881775, -1.4665415287017822, -2.5591821670532227, -1.7227208614349365, -1.9411981105804443, -0.6351600885391235, -0.4901343584060669, -2.795569896697998, 0.7638842463493347, 0.8396270871162415, 0.00902595929801464, -0.12393735349178314, 0.12674790620803833, -0.42245370149612427, -0.4326540231704712, 0.015371018089354038, -2.2854535579681396, -0.2628270089626312, -0.7500019073486328, -0.23841658234596252, 0.2243705540895462, -0.7239709496498108, 0.42232537269592285, -0.5308183431625366, 0.8377034664154053, 1.4065862894058228, 0.46460652351379395, 0.6695255041122437, 0.760313093662262, -1.7869828939437866, -0.10028824955224991, 0.5221568942070007, -1.1726855039596558, 0.20838554203510284, -1.5919928550720215, 2.084244966506958, -0.19363822042942047, -0.5111606121063232, 0.08353616297245026, 1.2497152090072632, -0.1613016575574875, 0.03640996664762497, 0.0049526323564350605, -1.1946812868118286, -0.5753989815711975, -1.398276925086975, -1.191027045249939, -2.105088710784912, 0.4700067937374115, -0.5604085326194763, -0.2576602101325989, 1.146822452545166, -1.4600571393966675, 0.8137363791465759, -0.8028762340545654, -0.8676528930664062, 1.732672929763794, 0.947446882724762, 0.32431599497795105, -0.6424906849861145, -0.6155226826667786, -0.5886826515197754, 0.8062996864318848, 0.4055524468421936, 2.4737491607666016, 1.5177055597305298, -1.307655930519104, 1.9308454990386963, -0.30604803562164307, -0.09333787858486176, -0.26205381751060486, 0.8224525451660156, -1.6184449195861816, 1.6869969367980957, 0.9010255336761475, 1.0066310167312622, -1.3429723978042603, 0.0027734648901969194, -1.3643301725387573, -0.25835269689559937, -0.7944329977035522, -0.4937669336795807, -0.07916244864463806, -0.008275710046291351, 1.2424135208129883, -0.5062828063964844, -2.299887180328369, 0.05082736909389496, 0.7562214732170105, -0.8134890794754028, -0.36255842447280884, -1.5285420417785645, -0.427247554063797, 1.4247620105743408, -0.531554102897644, 1.9614087343215942, 0.6258372068405151, 0.6707819700241089, -0.8797581791877747, 1.2182201147079468, -0.07398206740617752, -1.8298530578613281, 0.6221688985824585, 0.41290515661239624, 0.6534728407859802, -0.9603075981140137, -1.20545494556427, -0.4212746024131775, 0.704857587814331, -0.19142448902130127, 0.6729073524475098, 1.3306316137313843, -0.4266726076602936, 0.9355361461639404, -1.4966233968734741, -0.4506094753742218, 0.19421912729740143, -0.6180731058120728, 0.9442035555839539, 0.43302470445632935, -2.0593464374542236, 0.372964084148407, 1.120672583580017, -0.6306939125061035, 0.8302950263023376, 1.2310066223144531, -1.0584285259246826, 0.2766942083835602, -1.0371888875961304, -0.2610064446926117, 0.6299363970756531, -1.2649065256118774, -0.9018260836601257, -0.04865105077624321, 0.7115558981895447, -0.6673223376274109, 0.8707425594329834, -0.346855491399765, -0.07391975820064545, -1.8008761405944824, 0.6348689198493958, -0.18375980854034424, -0.009631067514419556, -1.399933099746704, -0.36711764335632324, 0.6259492039680481, -0.25447699427604675, -1.4444999694824219, 0.27480456233024597, -0.45791393518447876, 0.29809650778770447, 1.642655611038208, 0.06055501103401184, 0.8722245097160339, 0.2560769319534302, -1.0265783071517944, -2.189798593521118, -0.699084997177124, -1.0072699785232544, -1.2451527118682861, 1.7161695957183838, 0.6861771941184998, 2.7444424629211426, -0.4462278187274933, 0.10690239816904068, -0.7720953226089478, 1.9406605958938599, -0.2705845236778259, -0.4223269522190094, 1.002409815788269, 1.3476412296295166, -0.9886820912361145, -0.4279963970184326, -1.5268021821975708, 1.1085165739059448, 0.38260239362716675, 0.37557733058929443, -1.6106473207473755, 0.5000226497650146, -0.043689560145139694, 0.22749702632427216, 0.25246375799179077, 0.060465820133686066, 1.1725178956985474, 1.1300220489501953, 1.063327670097351, -0.7290065288543701, -0.27015969157218933, -1.2179514169692993, -2.0555613040924072, 0.8249700665473938, 0.3464786112308502, 0.13735345005989075, 0.484775185585022, 0.39063510298728943, 0.7287228107452393, -0.6653721332550049, 1.156220555305481, 1.2372007369995117, 0.22527863085269928, 0.09048585593700409, 0.5411481857299805, -0.5846213698387146, 0.3775409162044525, -0.4795200824737549, 0.08379704505205154, 0.725071132183075, 0.1390610635280609, -1.8156700134277344, -1.4615635871887207, -0.3621166944503784, -2.329071283340454, -1.9108537435531616, -0.12677519023418427, -0.6876402497291565, -0.8973521590232849, -0.4499613344669342, -1.131335735321045, 0.6378316283226013, -0.9937260746955872, -0.43665197491645813, -0.4503398835659027, 0.04519880563020706, 0.12323695421218872, 1.1352672576904297, 0.6892803907394409, 0.48920688033103943, 0.8972622752189636, 1.6177842617034912, 1.2687640190124512, -3.3888111114501953, 0.21160739660263062, 0.3666098713874817, 0.5670745968818665, 1.4698760509490967, -0.36254334449768066, -0.6407049298286438, -1.5727611780166626, -0.48576924204826355, 0.8615884780883789, 0.44065600633621216, 1.0930986404418945, -0.12566205859184265, -0.13785041868686676, -0.7331226468086243, 1.1511744260787964, -0.6088367700576782, -1.5942580699920654, 1.2301956415176392, -0.21910393238067627, -1.563233733177185, -0.18571406602859497, -0.38655886054039, -0.08127670735120773, 0.8362634778022766, -0.842458188533783, 0.795039176940918, 0.9506961703300476, -1.7940329313278198, -0.6899712681770325, 0.3013003468513489, -1.313028335571289, 0.4532390832901001, 0.4646489918231964, -1.7318634986877441, -0.8523666262626648, 1.4172638654708862, -0.5481677651405334, 0.7824380993843079, 0.15716683864593506, 0.22362270951271057, -0.758522093296051, 0.48337656259536743, 1.582026720046997, -0.7006762027740479, -0.709434986114502, 0.16267818212509155, 0.0741937905550003, 0.22159388661384583, -0.13888229429721832, -1.076560616493225, -0.0727284774184227, -0.21001118421554565, -0.3495416045188904, -2.020128011703491, -1.5726146697998047, 0.6387383341789246, -0.9696714282035828, 0.6563112139701843, 1.4113132953643799, -0.17721286416053772, -0.32355210185050964, -0.23136596381664276, -0.6852166652679443, -0.46530571579933167, -0.6888458132743835, 0.2272535264492035, 0.016328923404216766, -1.8970319032669067, 0.6350859999656677, -0.4990292191505432, -0.3885357677936554, -0.7977992296218872, -1.4896013736724854, 1.1001981496810913, 0.8950501680374146, 1.0704182386398315, 0.8505547642707825, 0.7952979803085327, 0.1149483248591423, 0.4475442171096802, 2.1489648818969727, -0.8638673424720764, 2.228283643722534, 0.5114620923995972, -0.8008760809898376, -0.1221231147646904, -0.7642241716384888, 0.6282120943069458, 0.4160533547401428, -1.4935805797576904, -1.031660556793213, 1.5456254482269287, 0.1436067372560501, -0.6566778421401978, -1.4150911569595337, -0.706527829170227, -2.869314432144165, -0.5057129263877869, -0.3466838002204895, -2.099860429763794, 0.6931509375572205, -1.6948986053466797, -0.43358469009399414, 2.215301752090454, -0.6064236164093018, -1.1760200262069702, 0.023857517167925835, -0.5509151220321655, -0.3193403482437134, 0.7570527791976929, -0.23597176373004913, 0.06790803372859955, -0.49745890498161316, 0.9556933045387268, 0.4518192708492279, 0.3612339198589325, -0.773942768573761, 0.08906449377536774, -0.05600091814994812, -0.4599178731441498, 1.2080074548721313, -0.6922387480735779, 0.95993971824646, 0.37556418776512146, 1.1404168605804443, -1.6043204069137573, -0.764972448348999, 0.5050493478775024, 0.4091334342956543, -1.1223701238632202, 2.1041572093963623, -0.8482211232185364, 1.0905169248580933, -0.7743323445320129, 1.500839114189148, 0.016193609684705734, -0.5057440400123596, 1.7420109510421753, -1.3548022508621216, -0.22310496866703033, 1.0939230918884277, -0.9556517601013184, -2.0512397289276123, -1.660522222518921, -2.0798380374908447, -1.3261407613754272, -0.10523025691509247, -0.5127771496772766, 0.3559700846672058, -1.0665420293807983, 0.8853936791419983, -0.36139115691185, -0.12377931922674179, 2.008124828338623, -1.020984411239624, 0.219465970993042, -1.7334988117218018, 0.697492241859436, 0.7768253684043884, -0.3779318928718567, -1.7423362731933594, -1.1444240808486938, -0.67567378282547, 0.8236680626869202, 0.504375696182251, -0.01773643307387829, -0.10880211740732193, 1.4795808792114258, -1.0684123039245605, 0.5726373791694641, -1.6925914287567139, -0.8765170574188232, 0.5347950458526611, 0.36141490936279297, 2.453679084777832, -0.03552532568573952, 1.532672643661499, -0.5629048943519592, 0.6940187215805054, -0.1361882984638214, -0.22023281455039978, -1.0979547500610352, 1.2689391374588013, 0.8754782676696777, 2.0870323181152344, -1.3305450677871704, -0.8092517852783203, -1.135419249534607, -0.7112256288528442, 0.14716389775276184, -1.6609222888946533, -0.49764859676361084, -1.2594237327575684, -0.5730650424957275, -0.18485242128372192, -1.0352569818496704, -0.5848556160926819, 0.3488021492958069, -0.5395217537879944, -0.0006422464502975345, -1.0521650314331055, -0.3796112835407257, 1.0393295288085938, 0.4802265763282776, 0.02657264471054077, 0.2662176787853241, 0.4945518672466278, -0.6113282442092896, 0.7369480133056641, 1.2579258680343628, 0.016859253868460655, 0.6303295493125916, -0.7360381484031677, -1.620667576789856, -0.852114737033844, 0.7413089871406555, 0.49202314019203186, 0.051669664680957794, -0.4447031319141388, -1.0315531492233276, -0.3362749516963959, 0.796006441116333, -1.6482938528060913, -0.6908984780311584, -1.3614827394485474, -0.38318780064582825, -1.7189205884933472, -0.04102116823196411, 0.0006055081030353904, -1.178329348564148, 0.24722734093666077, -0.6955437064170837, 1.2763701677322388, 1.2681589126586914, 0.3961378335952759, -0.26709580421447754, 0.2852359712123871, 0.41933193802833557, -0.6637102365493774, -0.7773653268814087, -0.950122594833374, -1.0496913194656372, 0.7461684346199036, 0.7330700755119324, -0.7395207285881042, 0.5528922080993652, 1.2026447057724, -0.24637727439403534, -2.043292284011841, 1.0976837873458862, -1.0947165489196777, 0.15056447684764862, -0.08599130809307098, 0.8977348804473877, 0.2388046830892563, -0.19637063145637512, -0.6630551218986511, 0.9007047414779663, -1.219651222229004, -0.5338029861450195, 0.6378380656242371, 0.929471492767334, 0.893680989742279, 0.21708567440509796, 1.3424311876296997, 0.14096452295780182, 0.19630977511405945, -0.1026570126414299, -1.235716462135315, 0.38937875628471375, 0.9713591933250427, 0.8259738683700562, -0.2917407751083374, 0.9828375577926636, -0.5668559074401855, -1.3859776258468628, 0.4408353865146637, -0.8356431722640991, -0.028434330597519875, 0.9549052715301514, 0.7159119844436646, 0.37978997826576233, 1.464081883430481, 1.0717695951461792, 0.45701882243156433, -0.4803268611431122, -0.8824578523635864, 1.7083568572998047, 0.7200537323951721, -0.9414389133453369, -0.5600638389587402, 0.09526442736387253, -0.71953284740448, 0.6346741318702698, -0.35797539353370667, -0.14057199656963348, 0.3898344337940216, -2.137742519378662, 0.13142573833465576, -1.9155066013336182, 0.17967167496681213, -0.4205567240715027, -0.43288424611091614, -0.753441333770752, 0.69342440366745, -0.4350254237651825, -1.1633886098861694, -0.23462650179862976, -0.29433387517929077, 1.1960606575012207, 0.7501514554023743, 0.8599387407302856, -0.9586097002029419, -0.8828989267349243, -0.2504979074001312, 0.5374847650527954, -0.24483604729175568, -0.5021727085113525, 0.31124812364578247, -1.0797141790390015, 1.448217749595642, 0.49941855669021606, -2.12221622467041, -0.8243045806884766, -0.36706578731536865, -0.8141701817512512, 0.8321946263313293, -0.3606656789779663, -1.0551135540008545, 0.10758422315120697, 1.3288549184799194, 0.6407034397125244, -0.10605786740779877, -0.8663139343261719, -0.2686108350753784, 0.12082912772893906, -0.69553542137146, 0.8333252668380737, -1.3699934482574463, -0.33446407318115234, -0.4943554103374481, -1.6338896751403809, 0.6763540506362915, 0.04403081163764, 1.8308731317520142, -0.8664759397506714, -1.396907091140747, 1.3359626531600952, 0.8843172788619995, -1.8910741806030273, -1.2779475450515747, 0.6275931000709534, -2.0689427852630615, -0.1377347707748413, 0.7308623194694519, 0.1312430053949356, 0.20361123979091644, -0.10648305714130402, 1.7390124797821045, 0.3661392629146576, 0.16856153309345245, 0.8756654858589172, -0.8365102410316467, 0.05341318994760513, 0.13063889741897583, -0.9115465879440308, 0.3548557162284851, -0.715499997138977, -1.068181037902832, 0.9073004126548767, 0.5449954271316528, 0.6343725323677063, -0.0873805582523346, 0.7512770295143127, 0.826152503490448, -0.5527229905128479, -0.25462400913238525, 1.16559636592865, 0.34073495864868164, -2.3207194805145264, -0.6540964841842651, 1.327183485031128, 0.11384818702936172, 0.4222151041030884, -0.5594795942306519, 1.3726710081100464, 0.45394808053970337, -0.7231451869010925, -0.29927149415016174, -0.4097221791744232, 0.044167354702949524, -0.050199609249830246, 1.2291666269302368, 0.19207604229450226, -0.7107046842575073, 2.8300909996032715, 0.19027231633663177, 2.2804641723632812, 0.026121754199266434, 0.8939520716667175, -0.5234531760215759, 0.5982829928398132, -2.4800994396209717, 0.2614730894565582, 0.08574996888637543, -1.4847962856292725, -0.9807772040367126, -0.14310988783836365, -0.6165953874588013, 0.6846621632575989, -1.5804736614227295, 0.5530804991722107, -0.12414338439702988, -0.6210377216339111, 0.11529874056577682, 0.534102201461792, 0.18940262496471405, -0.6479102969169617, 1.0110725164413452, 0.9034932255744934, -0.7210874557495117, -0.993837296962738, -0.3181084394454956, -1.0928893089294434, -0.03021712228655815, 0.35465118288993835, 0.5699746012687683, -0.37832263112068176, 0.6100203394889832, 0.8221448659896851, 0.5721869468688965, -0.5798451900482178, -0.2314823418855667, 0.6781426072120667, -0.9322693347930908, 0.3613335192203522, 1.5495010614395142, -0.36250925064086914, -0.10622815042734146, -0.8995895385742188, -2.015507221221924, -1.9429408311843872, -1.0204577445983887, -0.8789479732513428, -1.8256503343582153, 2.5031933784484863, 0.4860737919807434, -0.03341682255268097, -1.2683500051498413, -0.6528303027153015, -0.5428586006164551, -1.2100284099578857, 0.7780191898345947, 0.4586719274520874, 2.0546040534973145, -0.06831862777471542, -0.6577938199043274, -0.4336293935775757, -0.39448103308677673, -1.2570124864578247, -0.3821377456188202, 1.232027530670166, -1.8167346715927124, 1.1118731498718262, 0.6258032321929932, 1.1600648164749146, 0.5323930978775024, -0.6627908945083618, 0.9772309064865112, 0.03086603619158268, -0.9867763519287109, -0.04034218192100525, -0.4416772425174713, -0.002446526661515236, 1.0650607347488403, 1.7482125759124756, 1.193631887435913, -1.60845148563385, -1.0585739612579346, -0.5975327491760254, -1.1667605638504028, 0.3003230392932892, 0.40093982219696045, -0.10434450209140778, -1.0471217632293701, 1.0400567054748535, 1.4590911865234375, -1.7757548093795776, 0.9542760848999023, -0.0035260049626231194, 0.9154844284057617, 0.1961374133825302, -0.6525009274482727, -0.25820907950401306, 2.0710020065307617, -0.4439047574996948, 0.05849077180027962, 0.8199142813682556, -0.9417504072189331, 0.43318551778793335, 0.15961748361587524, 0.8560569882392883, -0.9604262709617615, 0.7088512182235718, 0.5649563670158386, 1.7990031242370605, -1.867485523223877, 0.8068954944610596, 0.4748723804950714, 0.41791215538978577, 1.3631218671798706, 1.3227059841156006, 2.0942800045013428, -0.4406929016113281, -0.268256813287735, 0.46711161732673645, 1.0030232667922974, -0.2507553696632385, 0.8122392296791077, 1.7005136013031006, -0.434659481048584, -0.0074873799458146095, -0.10315322875976562, -2.2185261249542236, 0.395656555891037, 0.31746163964271545, -1.8473529815673828, 0.16316665709018707, -0.28346729278564453, 0.0031914811115711927, -2.2574622631073, 0.33580440282821655, 0.6310560703277588, 0.2010076940059662, 0.42755964398384094, 1.3222780227661133, 0.8097108602523804, 0.5953588485717773, 0.20892754197120667, -1.4845010042190552, -1.160744309425354, 1.4973628520965576, 2.148888349533081, 0.9834170341491699, 0.20114123821258545, -0.26394790410995483, 0.3721999526023865, -0.1664239764213562, -0.7665368318557739, -0.10570218414068222, 1.526793122291565, 0.06895337998867035, -1.4405536651611328, -0.5646149516105652, 0.7529828548431396, 1.2235180139541626, 1.4787601232528687, -0.07690202444791794, -0.23830538988113403, 0.706743061542511, 0.5895881652832031, -0.4950418472290039, 1.5320689678192139, 0.20780231058597565, 0.6564401388168335, 1.7745071649551392, 1.1157258749008179, -1.0140812397003174, -0.8974270224571228, -0.8375679850578308, -2.173705577850342, -0.4970896542072296, 2.3097338676452637, -0.33926093578338623, -0.03276779130101204, 0.30263450741767883, 0.5504243969917297, -1.6122617721557617, -0.417968213558197, 1.0483194589614868, 0.8052443265914917, 0.7953965663909912, -2.65885853767395, -0.24821102619171143, -2.182387351989746, 1.6358588933944702, 1.2602477073669434, 1.0370073318481445, 0.05654680356383324, -0.0527651272714138, -0.9529178738594055, 0.28086337447166443, 0.9435580372810364, -1.1705679893493652, -0.7217311859130859, 0.5859059691429138, -1.8367500305175781, 0.2651691436767578, -1.085736870765686, -1.1706854104995728, 0.1607189178466797, -0.8039505481719971, 0.40864381194114685, -0.8844224214553833, 0.7557531595230103, 2.143333911895752, 0.5976115465164185, 0.0877644419670105, -1.1477746963500977, 1.5264344215393066, 1.1499062776565552, -0.6382303833961487, 0.19031758606433868, 0.9194315671920776, -1.3372324705123901, -0.23278050124645233, -0.8924890756607056, 0.2276650071144104, -0.13116353750228882, -0.7900493741035461, -0.5200327634811401, -1.2058368921279907, -0.6041540503501892, -1.364533543586731, 1.3526182174682617, -0.16066215932369232, 0.9104948043823242, -0.30589962005615234, 0.30604085326194763, -0.24378855526447296, -0.2853894829750061, 0.06062348559498787, -0.21374963223934174, 0.45048201084136963, -0.9275113344192505, -0.1516450047492981, 1.4925214052200317, -0.34787648916244507, 0.28531137108802795, 0.5901384949684143, 0.5566897392272949, -0.744278609752655, 0.2861756682395935, -1.8953839540481567, 0.3946756422519684, -0.5389743447303772, 0.5513537526130676, 1.205159306526184, -1.631481409072876, 0.6996835470199585, -1.4873688220977783, -0.5350890159606934, -0.4452607035636902, -1.140323281288147, -1.0621181726455688, -0.43533220887184143, -0.7983221411705017, -1.1445029973983765, -0.6622394323348999, -0.04316260665655136, 0.4135265052318573, -0.7800843119621277, -0.9266696572303772, 0.552749752998352, 0.44311559200286865, 0.569141685962677, -1.8177900314331055, -0.03712881729006767, -0.6862730383872986, -1.6499156951904297, -0.7160762548446655, -0.08373136073350906, -0.47967296838760376, 0.8386602997779846, -0.41592809557914734, -1.2263219356536865, 0.8827924132347107, 0.08397407829761505, 0.3852156698703766, 2.6208655834198, 0.41792675852775574, 0.3174818456172943, -0.30390578508377075, -1.1866933107376099, 0.7557634711265564, 0.13307884335517883, 0.22235208749771118, 1.1124128103256226, -1.4852617979049683, 0.4977057874202728, -1.1717242002487183, 1.1042444705963135, 0.224125936627388, 0.40232568979263306, 0.5910391807556152, 0.03939050808548927, 1.9166219234466553, -0.4649759531021118, -0.6597043871879578, 0.3634617328643799, 0.9431037902832031, 0.07327508926391602, -0.20418719947338104, 0.011190949007868767], [1.4836066961288452, -0.8504270315170288, -0.776879072189331, -0.7199875116348267, -0.05570770800113678, -1.1611018180847168, -0.680101752281189, 0.8067230582237244, -1.596958875656128, 0.06268521398305893, 0.9561780691146851, 0.7954791188240051, -0.1236121878027916, -0.5924999117851257, -0.15307696163654327, -0.6811351180076599, 1.6000697612762451, 0.056319862604141235, 1.4145587682724, -1.2910834550857544, -0.046845320612192154, 3.362140655517578, -1.4819695949554443, -0.45426490902900696, -0.5377895832061768, 1.6041090488433838, -0.15291191637516022, 1.1934807300567627, 0.3278966248035431, -1.1439425945281982, 0.3903006613254547, 0.728643536567688, 1.5590026378631592, -0.7030282616615295, -0.20260994136333466, 2.8055520057678223, -0.24428649246692657, 0.3993082046508789, 0.9306094646453857, -0.8997724056243896, -0.1742476224899292, 1.4573839902877808, -1.373295783996582, -1.3740520477294922, 0.3727380633354187, -0.3750206232070923, 1.3760892152786255, -0.6334909200668335, 1.6349142789840698, -1.495855450630188, -1.3323416709899902, 1.0679477453231812, 0.19725216925144196, -1.2771060466766357, -0.3661860525608063, 0.05442775413393974, -0.5911424160003662, -0.482709676027298, -2.1183016300201416, 1.1003862619400024, -1.0756678581237793, 2.246814489364624, 0.21904173493385315, 1.3125169277191162, 0.8285011053085327, -0.3561505079269409, -1.0388492345809937, 0.5664544701576233, -0.7237687110900879, -1.5041275024414062, -1.186537504196167, -1.583729863166809, -0.6038020849227905, -0.19350433349609375, -1.3496181964874268, -0.2858002781867981, -0.425775945186615, 0.3032854199409485, -1.8607714176177979, 1.7414181232452393, 0.31763556599617004, -0.8241516947746277, 1.5354070663452148, 0.8786194324493408, -0.058670781552791595, -0.546612560749054, -0.27054470777511597, 0.4884495437145233, 0.2372572273015976, 0.6564153432846069, -0.17513805627822876, -0.7397881746292114, -0.6498174667358398, 0.40072551369667053, 0.07222017645835876, 0.729074239730835, -0.7235298156738281, 1.527097463607788, -1.543143391609192, 0.20532897114753723, -1.7624680995941162, 1.093090534210205, 0.972528874874115, 0.23549434542655945, -0.08522714674472809, 0.018852323293685913, 0.3965373933315277, -0.5464919805526733, -1.6405243873596191, -0.5189957618713379, -0.6373055577278137, 1.6706362962722778, 2.133835792541504, 0.9376422166824341, 1.5734256505966187, -0.3490486741065979, 3.122159719467163, 0.3790242671966553, -2.110304832458496, 0.3809007406234741, -0.45489832758903503, -1.2008692026138306, -0.7919691205024719, 0.045162711292505264, -0.9189469814300537, 0.316577285528183, 0.3662114143371582, -1.6761345863342285, 1.0475963354110718, 1.383499264717102, 0.33902668952941895, 0.22154568135738373, -0.20790256559848785, -0.24415111541748047, -2.0875344276428223, -0.3220542073249817, 1.6115782260894775, 0.7655195593833923, 1.5256561040878296, -0.36391592025756836, 0.24036678671836853, 0.31575509905815125, 1.9803388118743896, -0.6352276802062988, -0.9209371209144592, 0.993331253528595, -0.310469388961792, 0.3588305413722992, -0.948644757270813, -0.18358996510505676, 0.7973621487617493, 0.687103271484375, 0.5546244382858276, -0.8132895231246948, 0.28769052028656006, -1.0313502550125122, 1.23479163646698, 0.10190405696630478, -1.5534560680389404, 0.5144039392471313, 0.4749929904937744, -1.4363439083099365, -0.6952742338180542, 0.4340357184410095, -0.694948136806488, 0.408563494682312, 1.4434598684310913, -0.19223453104496002, -1.7032326459884644, 1.353274941444397, 1.0889226198196411, -0.39429235458374023, 2.8570828437805176, 1.5609458684921265, -0.15195463597774506, -0.13048037886619568, -0.6504406929016113, -0.47203874588012695, 1.9132989645004272, 0.3755800724029541, 0.8526316285133362, 0.5135499238967896, -0.07783810794353485, 0.5709784626960754, 0.16395510733127594, -0.5710052847862244, -0.1986871212720871, 0.5194814205169678, -0.17623256146907806, -1.194090723991394, -0.39125311374664307, 0.5102011561393738, 0.17324307560920715, -0.8866884112358093, -1.398560643196106, 0.4393790662288666, -0.3494616746902466, -1.0541131496429443, 0.8489359021186829, 0.7677516937255859, 0.02751154825091362, 0.6573864221572876, -1.1078252792358398, 0.9990484714508057, 2.158345937728882, 0.7316603064537048, 0.7113396525382996, -0.45018836855888367, 1.815522313117981, -0.18974550068378448, -2.2400624752044678, 1.6800751686096191, 0.014833733439445496, -0.12058836221694946, 0.937775731086731, 0.08425340801477432, 0.6771605014801025, 0.19070807099342346, 0.5307121276855469, 1.0823183059692383, -1.2689512968063354, 1.771122932434082, -0.46704715490341187, 0.6294823884963989, -1.889053463935852, -1.1612380743026733, 0.24864420294761658, 1.3724812269210815, -0.920794665813446, -0.8839141726493835, -1.0352596044540405, 2.142857074737549, -1.9018126726150513, 0.6431978940963745, 1.7379722595214844, 2.2367656230926514, -1.015438199043274, 0.6304598450660706, 0.7298744916915894, 1.095646619796753, 0.16188232600688934, 1.1234104633331299, 1.2699017524719238, -1.9846209287643433, -0.807564914226532, -1.486099123954773, -0.6534715294837952, -0.4638635814189911, 0.8158502578735352, 1.4076210260391235, -0.08707502484321594, -0.22834603488445282, -1.0706815719604492, 0.14536981284618378, -1.5519499778747559, 0.37928998470306396, -1.139764666557312, 1.2089979648590088, 0.8967342376708984, -1.00412917137146, -0.8163639903068542, -0.7428102493286133, -0.7021250128746033, 0.9427573084831238, 1.7070029973983765, -1.1479616165161133, 0.868563711643219, -0.10617468506097794, 0.4289036989212036, 0.11414429545402527, -0.9498744010925293, 0.8792576789855957, -0.6686066389083862, 0.3321135342121124, -1.184794545173645, 1.0535632371902466, -1.5114551782608032, -1.5907634496688843, -0.4294011890888214, 0.05422800034284592, 0.8083668947219849, 1.3384284973144531, 0.16716836392879486, 1.150713562965393, 1.0874983072280884, -0.4211377501487732, 0.23645241558551788, -0.3997790813446045, -1.2596925497055054, 0.7181200981140137, -0.0992448702454567, 0.4371054768562317, 1.110272765159607, 0.6138438582420349, 0.030674826353788376, -0.45503610372543335, -1.2938315868377686, 0.2210894376039505, -1.0783555507659912, -0.6264499425888062, 0.1718384474515915, -0.043847642838954926, 0.8682844042778015, 1.6144464015960693, -0.48486101627349854, -0.7715559005737305, 0.3083258271217346, -1.026380181312561, -0.5366690158843994, 0.6281238794326782, 0.2523532509803772, -0.7681533098220825, 0.3177782893180847, 0.9371488690376282, 1.0026711225509644, -0.7579044699668884, -0.04836742952466011, 3.061504602432251, 0.7921346426010132, 0.21520690619945526, 2.3719379901885986, -0.23657025396823883, 0.2896241843700409, 0.16764673590660095, -2.1367976665496826, 1.40607750415802, -0.24078281223773956, -1.135202169418335, -0.4466540515422821, -0.3303074240684509, 1.349331259727478, 0.5994945764541626, -0.08281870186328888, -1.3282071352005005, -0.33289188146591187, 0.0501408576965332, -0.9426829218864441, 0.8957532644271851, 0.8956034779548645, -1.8231755495071411, -0.7445736527442932, 1.778480052947998, 0.32321539521217346, -0.19023607671260834, 1.8598642349243164, 0.045793309807777405, -0.05107267573475838, 0.5588218569755554, 2.4038021564483643, -0.06799478083848953, -0.10925760120153427, -0.26602160930633545, -0.7830905318260193, -0.05706217885017395, 1.472926139831543, 1.6620322465896606, 1.2469117641448975, -0.3496054708957672, -0.2455611675977707, -0.2037191241979599, -0.11912238597869873, -0.9928288459777832, 0.02205866016447544, 1.3558696508407593, 0.5943699479103088, 2.1375269889831543, -0.28269192576408386, 0.04370036721229553, -1.1442556381225586, 1.1133785247802734, 0.46213531494140625, 0.5194141864776611, -1.1960004568099976, 0.9040203094482422, -0.4213548004627228, 1.7430866956710815, -0.7211870551109314, 0.14180585741996765, -0.001325219520367682, -1.2802287340164185, 0.9983072280883789, -0.389382928609848, 1.0949368476867676, -1.8227674961090088, -1.214164137840271, -0.9103603363037109, 1.663814663887024, -0.06826989352703094, -0.3757689893245697, -0.6986985206604004, -0.11749278008937836, 1.1830768585205078, -1.7771178483963013, -0.5245612263679504, 0.7611270546913147, -0.909732460975647, 0.9176234602928162, -1.664603590965271, -1.4401323795318604, -0.5634293556213379, 0.28369811177253723, -1.6943564414978027, -0.684350848197937, 0.3202224671840668, 1.0438958406448364, -1.6385297775268555, -0.031796813011169434, -1.284558653831482, -1.0915307998657227, -0.8251113295555115, 1.8530514240264893, -0.434619665145874, 0.5396997928619385, 0.9932986497879028, -1.7644363641738892, 0.6832762360572815, 0.9809359312057495, -0.5264450311660767, 0.4355323314666748, -0.28417152166366577, 0.7163769602775574, 1.5210509300231934, 0.0040151141583919525, -0.02591552771627903, 1.071914553642273, -0.008728309534490108, 0.4721757173538208, 0.867160975933075, -1.4286357164382935, -0.7741517424583435, -0.7144426107406616, -0.8330658078193665, 0.7912964820861816, 0.7559009790420532, 0.6561174988746643, 0.10247792303562164, 1.1548038721084595, 0.49835434556007385, -0.3409479558467865, 1.9439054727554321, -2.899423122406006, -1.3561043739318848, 0.4719471037387848, 1.435715675354004, 1.0441033840179443, 0.5254585146903992, -0.9881132245063782, -0.8647327423095703, -0.817094624042511, 0.2622089385986328, 0.20513209700584412, -1.1680428981781006, -0.3781517446041107, -0.01989917829632759, -1.0133854150772095, 1.2493996620178223, -0.21763527393341064, -0.25769472122192383, 0.515310525894165, -0.27144983410835266, 0.754025399684906, -0.041876234114170074, -0.848139762878418, 1.1695001125335693, 0.40626439452171326, -0.7221689820289612, -0.45167216658592224, 1.0040688514709473, -0.4433337450027466, 1.4789879322052002, 0.6690196990966797, 2.28161883354187, -0.7096477150917053, -0.8604463934898376, -0.3775149881839752, -0.5018063187599182, -1.1093707084655762, -0.2733546495437622, -1.5578641891479492, -1.075758934020996, 1.3210018873214722, 3.003542900085449, 0.9076297283172607, -0.9812098145484924, 0.2137708067893982, -0.3888981342315674, 0.19906264543533325, 1.7987658977508545, 0.1484367996454239, 1.0739341974258423, -0.5248926281929016, 1.5893006324768066, 0.520057737827301, 1.2539745569229126, 0.8184816241264343, -0.8426629304885864, 1.4689140319824219, -1.6100953817367554, -0.20863424241542816, 0.8065995573997498, -0.4989849030971527, -0.04643337056040764, 0.8709840774536133, 0.4173569977283478, -0.24007536470890045, 0.9609279632568359, -1.3377524614334106, -0.4134170711040497, 2.5583770275115967, 0.31592509150505066, 1.582268476486206, 0.615132212638855, 0.40831440687179565, 0.2613939046859741, -0.3867700397968292, 0.602102518081665, -1.188484787940979, 2.573032855987549, 1.755171298980713, 0.263997882604599, 1.2751103639602661, 0.8289112448692322, -0.10634522885084152, -0.7448769807815552, -0.6507925987243652, 0.5821271538734436, 1.3778492212295532, -1.2328215837478638, -0.23709002137184143, 0.09466571360826492, -1.0624823570251465, 2.187407970428467, -0.07832301408052444, 0.832281231880188, -0.4237147569656372, -1.5200626850128174, 0.47493138909339905, 0.08115576207637787, 0.043317779898643494, 1.7358887195587158, 0.04274924471974373, 0.7528483271598816, -2.513319492340088, 1.2437900304794312, 0.2875477969646454, 0.6397752165794373, 0.024570606648921967, 2.38740873336792, -0.20048688352108002, 0.25902408361434937, 2.7601308822631836, -1.0450832843780518, 0.8150811791419983, 1.0438674688339233, 1.4232076406478882, 0.12411656230688095, -0.40181881189346313, -0.46518781781196594, -0.6841424107551575, 0.5538858771324158, 0.3756992220878601, 1.1726093292236328, -0.07879021018743515, 1.387498140335083, -0.21579492092132568, 0.5900765657424927, -0.9805395007133484, -2.5063936710357666, -0.4428146779537201, 0.5639665722846985, 1.8639202117919922, -0.9772502183914185, 1.079774260520935, 0.08597999066114426, -0.04510565847158432, -0.19111023843288422, -0.4033133089542389, 0.6753970384597778, -0.19787655770778656, -1.0491358041763306, -0.23953117430210114, 1.3067151308059692, -0.7978333234786987, 1.0039644241333008, 0.5208296179771423, -1.2925416231155396, 0.21572624146938324, 1.0647891759872437, -1.4911692142486572, 0.6699070334434509, -0.17298570275306702, -1.4853172302246094, -0.8927715420722961, 0.3402601480484009, 0.23002104461193085, -1.8054721355438232, -0.4711659848690033, 0.6329812407493591, -0.07453221082687378, -1.0478053092956543, 0.46707314252853394, -0.6908424496650696, 0.4995433986186981, 0.4790163040161133, -0.09332024306058884, -0.07458623498678207, 0.9353151321411133, -1.0796562433242798, 0.8959839344024658, 3.1209330558776855, -0.8630167245864868, 0.018503613770008087, 1.2933381795883179, -0.36246392130851746, -0.9763510823249817, -0.29814085364341736, -1.424757719039917, -0.6794552206993103, 1.9045696258544922, -0.07782496511936188, -1.1440593004226685, -2.0879228115081787, 1.377109408378601, -0.1866116225719452, -1.0372319221496582, -1.1839748620986938, -1.132887601852417, 1.021357536315918, 0.19622790813446045, -0.29666540026664734, 0.9073846936225891, -1.081642985343933, -0.375473290681839, -0.5704836249351501, 0.5914755463600159, 0.41593340039253235, -0.4233655333518982, -0.11740019172430038, -1.8510128259658813, 1.2165353298187256, -0.24221335351467133, 0.782133162021637, -1.3186594247817993, -0.9161638617515564, 0.4737125039100647, -0.1700786054134369, -0.004137621261179447, -0.25101953744888306, 0.2841337025165558, 1.343929409980774, 1.808718204498291, -0.42564859986305237, -1.150809407234192, -1.713664174079895, 0.11050824075937271, -1.8088068962097168, 0.9712690711021423, 1.0071821212768555, -0.7895044684410095, 0.0127876503393054, -0.31806036829948425, -1.3194695711135864, -0.9656440615653992, 0.5952714681625366, 0.6312799453735352, -0.46017760038375854, 0.6580743193626404, 0.732489824295044, 0.8401791453361511, -1.7324146032333374, -0.47692492604255676, 0.23363064229488373, 0.6611338257789612, 1.250637173652649, 0.18630842864513397, -0.35288316011428833, -0.7350959777832031, -0.4867135286331177, 0.48314520716667175, -1.2632660865783691, 1.6581143140792847, 0.028927672654390335, -1.123100996017456, 0.3029156029224396, -0.33008700609207153, -1.3520417213439941, -1.3213459253311157, 1.1600934267044067, -2.6263513565063477, 1.67377507686615, -1.3451683521270752, -1.6512422561645508, 0.43692922592163086, -1.0317354202270508, -0.5286056995391846, -1.0558453798294067, -0.5339416861534119, 0.5296044945716858, 1.6740124225616455, 1.273423671722412, -1.6094870567321777, 1.2278895378112793, 1.551708459854126, -1.4303895235061646, -1.0478140115737915, -0.9444979429244995, -0.05620601028203964, 0.21678251028060913, 0.4686853289604187, 0.7005126476287842, 1.0114941596984863, -0.13805867731571198, -1.6267019510269165, 0.24856574833393097, 0.6753255128860474, 0.32639092206954956, -1.2776312828063965, -1.0818827152252197, 0.48103392124176025, -2.2885775566101074, -0.14105217158794403, 0.14248332381248474, 0.2335626780986786, -2.242483615875244, -0.08002351969480515, 0.8010950684547424, 1.6475319862365723, 0.520694375038147, 0.8300125598907471, 2.0323023796081543, 0.3520587384700775, 0.8477445244789124, -0.8612308502197266, 0.44726720452308655, -0.998684823513031, 1.263548731803894, 0.0770786926150322, 0.8835481405258179, 2.3332197666168213, -0.07342027127742767, 0.5222164988517761, 0.2736149728298187, 0.4240975081920624, -0.8640270829200745, -1.4882652759552002, -0.4109814167022705, 0.9615564346313477, 0.6877336502075195, -0.49476000666618347, -0.9091112017631531, -1.1568583250045776, 0.15837348997592926, -0.5675063729286194, -1.4367561340332031, -0.40581512451171875, -2.982619285583496, -0.31622081995010376, 0.35140717029571533, 0.7276179790496826, 0.26250219345092773, 0.8481521606445312, -0.364061564207077, 0.05240064486861229, -0.04811250790953636, -1.1157386302947998, -0.6434201598167419, 0.7346709370613098, -1.5627574920654297, -1.3996995687484741, 0.9427390694618225, -2.895524501800537, 1.4978673458099365, 0.26677608489990234, 0.39410829544067383, -1.6056987047195435, 1.9671952724456787, 0.1254960149526596, 0.7037866115570068, 0.7712163329124451, -0.8697088956832886, -0.23449353873729706, 0.7966411709785461, 0.3247659206390381, 0.9940699338912964, 0.21318793296813965, 1.6362998485565186, 0.6760311126708984, 0.048833027482032776, 1.9995425939559937, 1.1875782012939453, -1.8912955522537231, 1.0212990045547485, -1.178733229637146, 2.171151876449585, 1.858141303062439, 0.6147820949554443, -0.12982922792434692, -0.06260057538747787, -0.0786503329873085, -1.5219565629959106, 1.1353750228881836, -0.30728164315223694, -1.3792099952697754, 0.26454591751098633, -1.075937032699585, -0.21174272894859314, 0.15404821932315826, 0.826455295085907, 0.6483950018882751, -0.5964465737342834, -0.21772150695323944, -0.5125539898872375, 1.1155884265899658, 0.39219900965690613, -1.0579007863998413, 0.4401635527610779, -0.34180137515068054, -0.4540663957595825, -1.1825639009475708, -1.9114019870758057, 1.1295444965362549, -0.41832074522972107, -0.5662922263145447, 1.857276201248169, -0.11214759945869446, 0.9648142457008362, 0.03906375542283058, -2.257291793823242, -0.12601988017559052, 0.40192899107933044, 0.2582685649394989, 1.3194248676300049, 0.19962354004383087, -1.0037680864334106, 2.0299651622772217, -0.20217539370059967, -0.021106498315930367, 1.2603737115859985, -1.202250361442566, 0.9071102738380432, -0.012450411915779114, 1.1260838508605957, -1.0584429502487183, -1.1468337774276733, 0.6250960230827332, 1.0376958847045898, 0.8344732522964478, -1.7606282234191895, -0.6123256087303162, -0.7260409593582153, -0.14555463194847107, -0.5214436650276184, 0.45210349559783936, -0.3051769733428955, 0.11412486433982849, -1.830174446105957, 0.17149563133716583, -1.5388820171356201, -1.2787362337112427, -0.5749954581260681, -2.4293158054351807, -0.004830563440918922, 0.25605103373527527, -0.37540915608406067, 0.11319006979465485, 0.7262529134750366, 1.3321056365966797, 1.1510957479476929, -1.5005884170532227, 1.332593321800232, -0.23026862740516663, -2.211054563522339, 0.4923560917377472, 0.8454989790916443, -1.3419464826583862, -0.08618270605802536, -0.7684414982795715, -0.14296779036521912, 0.20345187187194824, 1.1514874696731567, -0.8194534182548523, -0.00043780982377938926, -1.2087827920913696, 0.2646569311618805, 1.5191757678985596, -1.7597633600234985, -0.0545380562543869, -1.5544610023498535, 0.5773518681526184, 0.9672266840934753, 0.8265731930732727, -1.132683515548706, -1.6205006837844849, -1.3045202493667603, 0.2044190913438797, -0.3748057186603546, 1.049438238143921, -0.7873456478118896, 1.090622067451477, 0.4585224390029907, -0.06533759832382202, -0.7865373492240906, -1.3962675333023071, -0.24631324410438538, -1.2724380493164062, -1.5820586681365967, -0.33062902092933655, -0.29337620735168457, -0.6242348551750183, 1.957793116569519, 0.6016018986701965, -0.21106907725334167, -0.5814409852027893, 0.2501988112926483, -1.7842072248458862, -0.943384051322937, 0.33758288621902466, -0.6939088106155396, -0.5212576389312744, 1.356648564338684, -1.9653759002685547, 1.6522146463394165, 1.003309726715088, -0.4873940944671631, -1.2509580850601196, -1.077039122581482, 0.956803560256958, -1.3649704456329346, 0.7914060950279236, -1.876401424407959, 2.069257974624634, -0.11414327472448349, -0.7240859270095825, 0.4109532833099365, 0.07236199826002121, 0.14107288420200348, 0.6740983724594116, 0.36001747846603394, -0.3043394982814789, -0.1328040510416031, -0.5649546384811401, -0.23235823214054108, 0.2672024369239807, -0.7044986486434937, -0.6578765511512756, -0.3231065571308136, -1.2871339321136475, -0.18188612163066864, 0.35995543003082275, 0.7654901742935181, -0.550226092338562, 1.5257583856582642, 0.19418956339359283, -0.8178545236587524, -0.25747236609458923, 1.4122811555862427, 0.1596938669681549, 0.8242971897125244, 0.14337266981601715, -0.6369277238845825, -2.2584893703460693, 0.5238737463951111, -0.055362310260534286, -0.7181323766708374, -0.6387778520584106, 0.7406973242759705, -0.4218495190143585, -0.12866349518299103, 0.9643930196762085, 0.7601432800292969, -0.947959303855896, -1.0496915578842163, -0.30100706219673157, 1.5090267658233643, 0.32819247245788574, -0.8131488561630249, 0.5057612657546997, 0.7289512157440186, 0.197953999042511, 2.159271001815796, 0.5322859287261963, 1.2720102071762085, -1.798308253288269, 0.6060448288917542, 1.1805235147476196, -0.23529893159866333, 0.9583278894424438, 0.7782154083251953, -1.3134965896606445, -1.3012579679489136, 0.4209986627101898, 0.5204653143882751, 1.3516982793807983, 0.7657430171966553, 0.6360997557640076, 2.3703560829162598, 0.17262762784957886, -1.218912124633789, 0.5166180729866028, -1.7257306575775146, 0.27904272079467773, 0.3968774080276489, -1.0639805793762207, 0.6727635860443115, -0.8612366914749146, 0.37049728631973267, -0.8791201114654541, 1.4498099088668823, 1.327321171760559, 0.7799445390701294, -0.4320563077926636, 0.15833500027656555, -1.2297309637069702, 0.6142860651016235, 0.9877657294273376, -0.7344201803207397, -0.7038423418998718, -0.7698875665664673, -0.34614312648773193, -1.1463788747787476, -0.5773485898971558, -1.6488902568817139, -0.3133872449398041, -0.6200947761535645], [-0.06902145594358444, -0.6023063659667969, -1.018131136894226, 0.7183024287223816, -0.47090601921081543, -2.3382058143615723, -0.9368076324462891, -1.7081758975982666, -1.7318353652954102, 0.18149356544017792, 0.7091833353042603, 0.29333823919296265, -1.4583982229232788, 1.4201387166976929, 2.059471368789673, 0.19264563918113708, 0.11539541184902191, 1.1589821577072144, -0.46706169843673706, -0.682910144329071, 0.12873178720474243, 1.9652212858200073, -0.08161576837301254, -0.2164485901594162, 0.9625255465507507, 1.2697561979293823, -0.1489371806383133, -0.2577519714832306, 1.0393584966659546, 0.17247113585472107, 0.37383750081062317, 1.1189559698104858, 0.3235955834388733, -0.8012014031410217, -0.21201086044311523, -0.6890107989311218, 0.5182206034660339, -0.14116767048835754, -0.08926036953926086, -0.2852199375629425, 0.8826836347579956, 0.5007694363594055, 1.7479192018508911, 0.4663190543651581, -0.09649097174406052, 1.4936875104904175, 0.4886181950569153, 1.3670835494995117, -0.37094298005104065, 0.974007248878479, -2.269022226333618, -1.912243366241455, -0.5079618096351624, 0.061114534735679626, -0.4396674335002899, 0.9850255846977234, -0.2154323160648346, -0.37336087226867676, 0.8125861883163452, -0.03234996274113655, 0.7957945466041565, 0.27627310156822205, -0.15721549093723297, -0.6405121684074402, 1.4541094303131104, 0.4081944525241852, -0.4046086370944977, 0.4087238907814026, 0.8579038977622986, 1.7195240259170532, -0.5331459641456604, -0.2715229392051697, -0.6388513445854187, 0.8327181339263916, -0.3991095721721649, 0.22689689695835114, 0.059629201889038086, 0.309531033039093, 0.780360758304596, -0.8769193291664124, 1.5072786808013916, -0.5639353394508362, 1.2790663242340088, 1.1722818613052368, -0.185937762260437, -0.6215195059776306, 0.9447212219238281, -0.943091630935669, 0.9148185849189758, 0.7805147171020508, 0.8103353977203369, -0.0992349237203598, -1.1780778169631958, -0.35848283767700195, 0.0683387741446495, -1.3571802377700806, -0.3480463922023773, 0.16227740049362183, 0.3730575144290924, 2.1652238368988037, 0.5993455648422241, 0.8946160078048706, 0.6318117380142212, 0.3607187569141388, 0.6535489559173584, 0.3217148780822754, 0.44160786271095276, -0.21058781445026398, 2.1850807666778564, -0.7116068601608276, 1.4505786895751953, -2.00400972366333, 1.567931890487671, 0.8362879157066345, -0.03569569066166878, 0.7675421237945557, 1.3094415664672852, 0.8479502201080322, -0.7703012228012085, 1.2919954061508179, -0.007511012256145477, 0.6021729707717896, -0.9375911951065063, -0.8925136923789978, 1.2121684551239014, 0.5897706747055054, 1.6980931758880615, 0.16884101927280426, 0.9361090660095215, 0.7562539577484131, -0.8977100849151611, -0.13638179004192352, 0.5511272549629211, 0.8246580362319946, -0.08526160567998886, -0.9025106430053711, -0.778343915939331, -1.182669758796692, -0.6420508027076721, 0.9662783741950989, -0.8359678387641907, 1.5289803743362427, -0.22605162858963013, -0.6371229887008667, 1.529477834701538, -0.2912178635597229, -0.4375019371509552, 0.8031251430511475, 1.9564112424850464, 1.0193297863006592, -0.20181220769882202, 1.5033921003341675, -0.5139431953430176, 0.27143603563308716, -1.2392921447753906, 0.4702123999595642, -0.43123379349708557, -0.5876333713531494, -1.3458436727523804, -0.2761014699935913, -0.2649387717247009, 0.23628142476081848, 0.6266821026802063, 0.6945673227310181, 0.22178232669830322, 0.9674636125564575, -0.47810935974121094, 2.4754269123077393, -0.12954512238502502, -0.3108992278575897, 0.5629085898399353, 1.655312180519104, -0.7405673861503601, 1.1645777225494385, 1.2519629001617432, -1.633687138557434, -0.35848334431648254, -0.420184850692749, 0.2867060899734497, -0.46977147459983826, 0.7917973399162292, 0.44197362661361694, -1.5776891708374023, 0.6830878853797913, 1.0629727840423584, 0.37390410900115967, -0.5330055952072144, -1.916587233543396, -0.7877128720283508, -0.7569020986557007, 0.11508633196353912, 0.3721175491809845, -1.1944026947021484, 2.1850764751434326, 0.39865410327911377, 0.8298918008804321, 0.7997042536735535, -0.4004625380039215, -0.527362048625946, 0.999226987361908, 0.4247671961784363, -1.4333499670028687, 0.3110544979572296, -1.1795316934585571, -0.7080070972442627, 0.6164018511772156, 0.02781311795115471, -0.9881330728530884, -0.1031622365117073, -0.09245488047599792, 0.8072968125343323, 2.5643646717071533, 0.2991812825202942, 1.18408203125, 0.4630359709262848, 0.5727663040161133, 0.5831966996192932, -0.45721888542175293, -1.9364356994628906, -0.8961276412010193, -1.2697237730026245, -1.6916333436965942, 1.1801650524139404, -0.18490764498710632, -1.9618216753005981, 0.6341824531555176, 0.8901895880699158, 0.962372362613678, -1.0048143863677979, 1.1315579414367676, 1.4518176317214966, 0.28790178894996643, -0.1628425419330597, -0.26973918080329895, 0.35788920521736145, -0.2852204144001007, -0.45306703448295593, 0.0785728245973587, -0.2571806013584137, -0.420704185962677, -1.6617990732192993, 0.2270696759223938, -0.46571213006973267, -0.38653188943862915, 0.7412533760070801, -1.5799742937088013, -0.019324028864502907, 1.634993314743042, 1.2901620864868164, -0.2946620583534241, -1.626451849937439, -1.0026657581329346, 0.16221760213375092, -0.05453171208500862, -0.8832316994667053, -1.370239019393921, -0.08730573952198029, -0.25793883204460144, 0.32691439986228943, -1.3095812797546387, 0.7642703652381897, -0.33060431480407715, -0.5464614629745483, -0.32212087512016296, -0.8923726081848145, 1.0676283836364746, -1.3938757181167603, 0.3338877558708191, 0.23254618048667908, 0.42119351029396057, 1.6038988828659058, -1.012130856513977, -0.5393809676170349, -1.7480802536010742, -1.0436902046203613, 0.4304235875606537, 0.7444014549255371, -1.5194411277770996, 1.1231918334960938, 1.2482600212097168, 0.44689682126045227, -0.5856381058692932, -0.5791066288948059, -0.4156355559825897, 1.3614873886108398, 0.8877570033073425, -0.08015808463096619, 0.7706501483917236, -0.6214837431907654, 1.353973150253296, 0.4336419701576233, -0.9670349955558777, 0.6888368129730225, 0.0003105824289377779, -0.6046373248100281, -0.8042070269584656, 0.9857721924781799, -0.6625602841377258, 0.4288818836212158, 0.4697519838809967, 0.5364718437194824, -0.4036550521850586, -0.14198049902915955, -3.0658762454986572, 0.34442564845085144, 1.5036864280700684, -0.05819049850106239, -0.1645221710205078, -0.6703553199768066, 1.445829153060913, -1.411515712738037, 0.03674685209989548, 0.5449819564819336, 0.2589797079563141, 0.016428427770733833, 0.8811426162719727, -0.5805130004882812, -1.340169906616211, 0.4848756194114685, 0.1454148143529892, -0.9454686045646667, -0.5125163793563843, -1.0722249746322632, -0.6131758093833923, -0.38479435443878174, 0.690205454826355, 0.2986578941345215, -0.021361956372857094, 0.4467903673648834, 1.3873026371002197, 1.299859881401062, -0.036309801042079926, 0.25770092010498047, 0.07990197092294693, 0.4898504316806793, -0.8941594362258911, 1.7753349542617798, -0.931788980960846, 1.2197964191436768, 0.7818360924720764, -0.6369301676750183, 0.59376460313797, 0.9590843319892883, 0.15962675213813782, 0.287670761346817, -0.6117017865180969, -2.3443870544433594, -0.6380387544631958, -0.2780139744281769, -0.4511830806732178, -1.5579464435577393, -1.212604284286499, 0.5867213606834412, -0.2531662583351135, -0.22076170146465302, 0.5294053554534912, 0.28300192952156067, 0.44367682933807373, 0.26832064986228943, -1.0503987073898315, -0.9134770035743713, 0.19167658686637878, -0.5388563275337219, -1.3786048889160156, -0.6399711966514587, 0.3219640552997589, -0.7418732047080994, 1.4828767776489258, -2.0430965423583984, 0.4317364990711212, -0.5534331798553467, -0.2852350175380707, -1.1458762884140015, 0.08766695857048035, -0.31645646691322327, 0.6289178729057312, -1.707105040550232, 0.11680038273334503, -0.4399198591709137, -0.8032621741294861, 1.0014368295669556, -0.04068724066019058, -0.15432704985141754, 0.5581327080726624, -1.192856788635254, -1.8284319639205933, 0.9290382266044617, -0.8084524869918823, 0.4015985429286957, 1.2006734609603882, 1.0005712509155273, 0.49583199620246887, 0.2022552639245987, -1.434721827507019, 0.9621212482452393, 0.15958231687545776, 0.0016549897845834494, -2.3060715198516846, 1.3396484851837158, 0.30803582072257996, -0.2363259643316269, 0.9175637364387512, -0.6873251795768738, -0.784331202507019, 0.9167661070823669, -0.38636961579322815, -0.7287077307701111, 0.4164658486843109, 0.0639268010854721, 2.0516202449798584, 1.1331493854522705, -0.8289214968681335, 1.1887319087982178, 1.4063386917114258, -0.2122296392917633, 0.08809966593980789, -1.4681369066238403, 0.3181929588317871, -0.5337446928024292, 1.6234720945358276, 0.42251792550086975, 0.16238948702812195, -0.4934561550617218, 0.7652926445007324, -1.8168822526931763, 0.870002269744873, -0.8892464637756348, 0.79103684425354, -0.31994739174842834, 0.48423829674720764, -0.06902024149894714, -0.32542353868484497, -0.41666755080223083, -0.2732813060283661, 2.0052123069763184, 0.19313642382621765, 1.226995587348938, -0.482583224773407, 0.14512784779071808, 0.5366036891937256, -0.7352375984191895, -0.11787824332714081, -0.6895702481269836, -0.7901098132133484, 0.7540769577026367, 2.1808958053588867, -0.11237277090549469, 1.1062995195388794, 0.6599034070968628, 0.2290455847978592, 0.46039682626724243, -0.025654181838035583, 1.1718358993530273, 0.6429596543312073, -1.0281003713607788, 0.2982918620109558, 0.34193697571754456, -0.06992283463478088, 0.09620054811239243, 0.7254409790039062, 0.2447553128004074, 0.486920565366745, 0.9679754972457886, 0.4378141760826111, 0.3719768524169922, -0.5272331833839417, -0.07796657085418701, -0.35750138759613037, -2.5066044330596924, -0.09912336617708206, -0.9873972535133362, -0.4005320072174072, 0.2978135347366333, 1.0399949550628662, -1.4899489879608154, 1.6588214635849, 0.012589125894010067, -0.734045147895813, -0.1143357902765274, -0.6382705569267273, -1.2932220697402954, 0.3131195306777954, 1.0616264343261719, -0.6307576298713684, 0.24197328090667725, 0.22705179452896118, -2.353519916534424, 0.14699138700962067, 1.7318929433822632, 0.06779118627309799, 0.33160579204559326, 0.7388038039207458, 0.7357895374298096, -0.3863774240016937, 3.1593289375305176, 1.9604802131652832, 1.371452808380127, 0.06387171894311905, -2.269869327545166, -0.04659191519021988, 0.3191201984882355, -0.5393281579017639, 0.05836264789104462, -1.4578174352645874, 0.7497271299362183, 0.604856550693512, -0.4546237885951996, -0.7806435823440552, 0.5474646091461182, -0.2505156099796295, 1.3780056238174438, -0.9132118225097656, 1.0280134677886963, 1.1422303915023804, 0.03428223729133606, -1.2103677988052368, 0.7649753093719482, 0.37155210971832275, 1.0043118000030518, -0.6837013959884644, 0.7714923024177551, 0.6036728024482727, 1.3756235837936401, 1.9216727018356323, -0.6636608242988586, -0.6415008306503296, 0.06365939974784851, -0.0035413538571447134, -0.20118527114391327, -1.1654043197631836, 0.4524517059326172, 0.005488829221576452, 0.8874825239181519, 0.049141038209199905, 0.06949791312217712, 0.06366865336894989, -0.4956649839878082, 0.857685923576355, -1.3505499362945557, -1.0605275630950928, 0.024530211463570595, 1.133527398109436, 0.36804136633872986, -0.567891001701355, 1.4270331859588623, 1.4287258386611938, -2.426710605621338, -0.9754765629768372, -0.7339121699333191, 0.27806341648101807, 0.40487584471702576, 0.6249176263809204, 0.30740463733673096, 0.7425243258476257, 1.5597482919692993, -0.154416024684906, 2.3145053386688232, 0.8560787439346313, -2.245988368988037, 0.39212432503700256, 1.6789216995239258, 1.7351248264312744, -0.2790893316268921, 2.746457099914551, 0.655362606048584, 2.796154737472534, 0.6261984705924988, -1.302463173866272, -1.7813785076141357, 1.6786365509033203, 1.7073887586593628, 1.5021010637283325, 1.3023346662521362, -0.1825757920742035, -0.3327988386154175, -0.8258249759674072, 0.7563968300819397, 0.7917407751083374, -1.091927170753479, 0.24633575975894928, 1.4417424201965332, -0.49567654728889465, -2.611860513687134, 0.8564937114715576, 1.0502781867980957, -1.4108508825302124, 0.29621002078056335, -0.4243116080760956, 0.6642093658447266, 0.38904502987861633, 1.884297490119934, -1.5939114093780518, 0.5354191660881042, 1.395707130432129, -0.10753992944955826, 0.016571538522839546, -0.4510015547275543, 2.211669683456421, -1.8004875183105469, 0.27946969866752625, 0.07886608690023422, -0.36527717113494873, 1.0967024564743042, 0.9873713254928589, -0.002933287061750889, 0.3752988874912262, 1.2908421754837036, -0.06789291650056839, -1.9190537929534912, 0.42795559763908386, 1.0233607292175293, -0.7501584887504578, 1.5281224250793457, -2.3968114852905273, 0.6713776588439941, 0.3315849304199219, -0.545262336730957, -1.9782415628433228, 0.3523057997226715, -1.0214948654174805, 0.02298283576965332, 1.233133316040039, 0.3038734495639801, 1.3842540979385376, 1.2407679557800293, 0.3219912648200989, 0.1532699018716812, -1.4119558334350586, 0.6633918285369873, -0.04740196093916893, 1.5937472581863403, 0.16897648572921753, 1.0197169780731201, 0.9280238747596741, 0.5508574843406677, -0.5044805407524109, 0.3959003984928131, 1.2491594552993774, -0.42399635910987854, 0.7128266096115112, -1.678870439529419, -1.4119967222213745, -0.15087349712848663, -0.34699350595474243, -0.7569359540939331, 0.3249117434024811, 1.9678455591201782, 0.02612796239554882, -1.123398780822754, 0.17475740611553192, -0.42177656292915344, -1.772360920906067, -0.6132319569587708, -1.0758532285690308, 0.6719574332237244, 0.10714364051818848, -0.5784966945648193, -0.5090753436088562, 0.8562608361244202, 0.9363183379173279, 0.8218538165092468, -2.3555245399475098, -0.7938327193260193, -0.7627793550491333, 0.39833948016166687, -1.0735374689102173, 0.4538441300392151, 1.1603189706802368, -0.15425485372543335, 1.7085330486297607, -0.9865787029266357, 0.16919752955436707, 1.793757677078247, 0.7031534314155579, -2.2645349502563477, -0.46480733156204224, -1.6049885749816895, 0.7609887719154358, 0.18778866529464722, -0.7650298476219177, -0.8440729379653931, 0.8370851278305054, 0.6376797556877136, 0.28841933608055115, -0.2748554050922394, -0.14519576728343964, -1.6988638639450073, 0.2865418791770935, 1.274014949798584, 0.5476883053779602, -0.009438748471438885, 1.3353064060211182, 1.2151978015899658, -1.6492598056793213, 1.0703400373458862, 0.4667949080467224, -1.9815319319604896e-05, -0.12246578931808472, -1.4950923919677734, -1.257201910018921, 0.9207461476325989, 0.6799931526184082, -0.28100278973579407, -0.3490543067455292, -0.14168590307235718, 0.6108354926109314, -0.15161021053791046, -1.4885939359664917, -0.9957196712493896, 1.5229849815368652, 0.5637658834457397, 0.4215152859687805, 1.1463621854782104, -0.8213157057762146, -0.6457867622375488, 0.34675922989845276, 0.7525707483291626, -0.7211254239082336, 1.634377121925354, 0.3524693250656128, 0.8379621505737305, -2.222867727279663, -0.33252039551734924, 1.7662779092788696, 0.8570775985717773, -0.649600088596344, 0.8645496368408203, 0.5267980694770813, -1.0013506412506104, -0.4154738485813141, -0.05068415775895119, 0.18036821484565735, -0.5022684335708618, -1.9681265354156494, -0.24138501286506653, 1.2849537134170532, 0.3772134482860565, -0.10871746391057968, 0.7093791365623474, 0.48121801018714905, -1.2177839279174805, -1.253315806388855, 0.4957990348339081, 0.005753979552537203, -1.6892248392105103, -1.4089161157608032, -0.05312581732869148, -0.24589836597442627, 0.9464100003242493, 0.08417227864265442, -0.8148543238639832, 1.302720308303833, -0.008397664874792099, -0.25223293900489807, -2.2406182289123535, -1.0529721975326538, -1.679449200630188, -2.013615131378174, 0.33997100591659546, -0.1018325611948967, 0.566036581993103, -0.6636794805526733, 0.7916301488876343, 1.1117087602615356, 0.427962988615036, 0.12553741037845612, -0.2428382933139801, 1.0951621532440186, -0.7289426922798157, 1.3896228075027466, 0.5357909202575684, -0.9389508962631226, -1.949790358543396, -0.6394845843315125, -0.4958621859550476, 2.172778844833374, -0.15400424599647522, -0.8529616594314575, 0.8819302320480347, -0.061236850917339325, 1.081076741218567, -1.1222323179244995, 1.619251012802124, -1.5261459350585938, -0.7574105858802795, 0.2837192118167877, 0.5348548889160156, 0.970110297203064, -0.33507004380226135, 1.4356133937835693, -1.8402823209762573, -0.23705655336380005, -0.22517536580562592, -0.07364924997091293, -0.14132378995418549, 1.4960464239120483, -0.7276285886764526, 1.3260389566421509, -1.8093504905700684, 0.8277717232704163, -0.9342909455299377, -0.5569947361946106, 0.6108938455581665, -0.44772523641586304, -1.738539457321167, 2.90588641166687, 0.6811457276344299, 0.8820971250534058, -1.8846405744552612, -0.13828998804092407, 1.1943767070770264, 0.8585460186004639, -0.22211281955242157, -1.891333818435669, -1.2799556255340576, -2.594139814376831, 0.6886943578720093, 0.24747279286384583, -0.27442601323127747, 0.4911777973175049, 0.20960626006126404, -0.626511812210083, 1.3103543519973755, 0.040862902998924255, -1.6738559007644653, -0.4513976573944092, -0.3807726800441742, -0.813666820526123, 0.20138081908226013, -0.5442209243774414, -0.2718910872936249, -0.6902086734771729, 0.33292946219444275, -0.4184330105781555, 1.499803900718689, 2.048381805419922, 0.8402706980705261, 0.615163266658783, -0.24195638298988342, 0.8797454237937927, -0.0027722560334950686, -1.57149338722229, -1.033828854560852, 0.06647884845733643, 0.34323185682296753, 0.20365376770496368, 0.7974803447723389, 0.3931589126586914, 0.6398746967315674, 0.44741660356521606, -2.1133320331573486, -0.32504063844680786, -0.9634204506874084, 0.3035222589969635, 1.6119024753570557, 1.356345534324646, -0.44144129753112793, 0.36069008708000183, 1.7865386009216309, -1.729524850845337, -0.7787585258483887, 0.7547304630279541, 0.739248514175415, 0.6545028686523438, -0.8933047652244568, 0.1964045912027359, 1.1861628293991089, 0.22134673595428467, 0.44203850626945496, 0.9997870326042175, 0.9279109835624695, 0.5114427208900452, 1.3035237789154053, -1.0861488580703735, 0.8473376631736755, 0.006897886283695698, -2.1242082118988037, -0.10243920236825943, 0.11675642430782318, -0.13278640806674957, -0.10337845236063004, 0.09547718614339828, 1.7731378078460693, -2.5646474361419678, -0.31430816650390625, 0.026340996846556664, -1.1434005498886108, 0.23436212539672852, 0.3382416367530823, -1.0241093635559082, 0.9117850065231323, 0.0346384197473526, -0.7556765079498291, 0.44368892908096313, -0.6667032241821289, -2.6359097957611084, -0.9968888759613037, -1.4111543893814087, -0.339966356754303, -1.7083371877670288, -0.01708799973130226, -0.1644149124622345, -0.0511123463511467, -2.754713296890259, -0.16970200836658478, 1.3203866481781006, 1.073966145515442, 1.751434326171875, 0.11632490903139114, 0.6136976480484009, -1.0091055631637573, -1.9832161664962769, 0.9741648435592651, -1.5147463083267212, -1.0305770635604858, 0.5775731801986694, -1.466670036315918, -0.6398122310638428, 0.8420681357383728, 0.8237671256065369, 0.5830695033073425, 0.32186654210090637, 0.33672207593917847, -1.7809175252914429, 0.00695966649800539, 0.021310564130544662, -1.8823882341384888, -0.2691488265991211, 0.8499813079833984, -0.1606796532869339, 0.22333280742168427, 0.7482624053955078, 0.6558311581611633, -0.1998511105775833, -0.1742653250694275, -0.5673525333404541, -0.2817341685295105, 1.346673846244812, -0.7368289828300476, -0.985481321811676, -0.15727780759334564, 0.5719104409217834, 0.31454744935035706, 0.27674925327301025, -0.8783769011497498, 2.3356380462646484, -0.6935274600982666, 0.4348721206188202, 1.4236565828323364, -1.1747820377349854, -0.4164976477622986, 0.3313419222831726, -0.5550529360771179, -1.1077885627746582, 0.29858630895614624, 0.01415208075195551, 0.7189539670944214, 0.22941553592681885, -1.4583797454833984, 2.349378824234009, 1.1239324808120728, 0.33556127548217773, -0.1041533574461937, -1.6364163160324097, -0.3102334439754486, 0.07716062664985657, 0.13092410564422607, 0.526192307472229, -0.10654769092798233, 1.7772575616836548, -0.07735205441713333, 0.3871392011642456, 1.6147825717926025, -1.000137209892273, -1.207111120223999, -0.48247483372688293, 1.22166907787323, -0.062166471034288406, -0.0560852475464344, -0.3616352379322052, -0.08671768754720688, 1.3481765985488892, -0.9613855481147766, 0.21014876663684845, 0.4432314336299896, -0.5127741098403931, -0.7552313208580017, 1.2693148851394653, -0.7929529547691345, -0.9288412928581238, 2.9195258617401123, -0.06110994517803192, -1.5624395608901978, 0.34644389152526855, 0.351816326379776, 0.07826951891183853, -0.4370669424533844, 1.2281241416931152, -0.3990151584148407, -1.3678778409957886, 0.9019320011138916, -2.2818198204040527, 0.2503547668457031, -0.1050015464425087, -1.794073462486267, 1.8050678968429565, 0.282443642616272, 1.7513681650161743, 0.4040575921535492, -1.5852572917938232, 0.2991637885570526, -0.8005840182304382, -1.6387858390808105, -2.124635696411133, 0.23498977720737457, -2.757249355316162, -0.5043706297874451, 0.2842921018600464, 0.017585011199116707, -0.2116582840681076, 0.3648700416088104, 1.1786167621612549, -1.1338183879852295, -1.4412598609924316], [-0.21923303604125977, -1.2583413124084473, 0.7385641932487488, 0.22633051872253418, -0.609521210193634, -0.06254879385232925, 0.8090304136276245, -0.11250784248113632, -1.59640634059906, 0.3804622292518616, -0.4392518699169159, -0.23281508684158325, -0.04542604088783264, 0.4932558238506317, -0.3194158673286438, 0.7161690592765808, -1.1584490537643433, 0.5765101909637451, 1.1262184381484985, 0.4070506989955902, -0.6872029900550842, -0.16419433057308197, 1.0388648509979248, 0.7307582497596741, -2.049715518951416, -0.47053831815719604, -1.875637412071228, 0.37644556164741516, -0.5101386904716492, 0.545924961566925, 1.2268497943878174, -0.7761940360069275, 0.7467209100723267, 1.7493170499801636, -0.118293896317482, -0.1502872109413147, -1.031247854232788, -1.5860286951065063, -0.42030778527259827, 1.3884354829788208, -1.4498882293701172, 1.1539021730422974, -0.5051003098487854, 0.8542929887771606, -2.0123555660247803, -1.0204776525497437, -1.16310453414917, 1.5317115783691406, 1.4660577774047852, -0.5580094456672668, 0.3229268193244934, 0.26501983404159546, 1.4476581811904907, -0.6536458134651184, -1.6078853607177734, 1.2486718893051147, 0.1823929101228714, -0.007628251798450947, -0.40865540504455566, -1.0838005542755127, 1.3055740594863892, -2.146517753601074, 1.6594414710998535, -0.25678470730781555, -1.917338490486145, 0.189681276679039, -1.3795329332351685, 1.1390485763549805, -0.40498849749565125, 0.4997476637363434, 0.4361897110939026, 0.9850957989692688, -1.625959873199463, 0.11472050845623016, -0.1810126155614853, -0.6877425312995911, -0.4402305781841278, -0.7336533665657043, 1.0776562690734863, 2.2157156467437744, -0.2025034874677658, -0.25907739996910095, -0.4872435927391052, 0.7448610067367554, 0.6873801350593567, 0.5540429949760437, -1.5046814680099487, 1.6756516695022583, -0.2345340996980667, -0.516775906085968, -0.6075337529182434, 0.6504239439964294, 0.6771094799041748, 0.738685667514801, -0.6505497097969055, -0.7492228150367737, 0.843174934387207, -0.18544261157512665, -1.1073198318481445, -1.5913246870040894, -0.7361331582069397, -1.4724411964416504, 1.7121105194091797, -0.4546872079372406, -1.6361804008483887, 1.5204615592956543, -0.7541730999946594, 0.980115532875061, 0.7302602529525757, 0.9195483326911926, -0.1427898108959198, -1.31635582447052, 0.509848952293396, -1.2431697845458984, -0.13776323199272156, -0.5895238518714905, 0.9871735572814941, 2.272007703781128, 0.8700407147407532, -0.09035743027925491, -1.4318246841430664, 2.095463514328003, 1.0740495920181274, 0.4463947117328644, 1.1293036937713623, -2.048414707183838, 0.10879210382699966, 0.06445804983377457, 0.2892985939979553, -0.16094490885734558, 0.25856873393058777, -0.7016836404800415, 0.8838856220245361, -1.0739904642105103, -0.8685936331748962, 0.44310832023620605, -1.5704960823059082, -1.028477430343628, -0.821553647518158, -1.652820110321045, -1.3416130542755127, 0.4361773729324341, -0.5603196024894714, 0.11571799963712692, 1.2316639423370361, 0.31377437710762024, 1.0363636016845703, -0.20818591117858887, -0.6383235454559326, -0.9391414523124695, 1.4238008260726929, 0.759759783744812, 0.3262685537338257, 1.0066372156143188, 1.22360360622406, 0.15595871210098267, 0.8525294065475464, 1.3209689855575562, 0.7107356190681458, 0.5827297568321228, -1.242012858390808, 2.304394483566284, 0.36572957038879395, -0.02202126383781433, -0.15999570488929749, 2.1146981716156006, -0.8192108273506165, -2.3685083389282227, 1.2269608974456787, -0.3437192738056183, -0.998611330986023, -3.3411307334899902, 0.15515592694282532, -0.1341560333967209, 0.3501715362071991, -0.309317409992218, -0.7015347480773926, 0.8977689146995544, -0.8418918251991272, 1.6674880981445312, 1.4528014659881592, 0.7360749244689941, 1.3241842985153198, -0.8858519196510315, 0.03942694887518883, 0.8430324792861938, 1.5096930265426636, -0.7001157402992249, -0.38079747557640076, -1.3816393613815308, 1.161881446838379, -1.089643955230713, -0.4999949038028717, 0.22463567554950714, -1.9018980264663696, -0.936633288860321, -0.25531837344169617, 0.01859777420759201, -0.37123265862464905, 1.0009222030639648, 1.3484894037246704, 0.951579749584198, -0.5610607862472534, -0.09017149358987808, 0.5715271234512329, -1.0810002088546753, 0.17956224083900452, -1.4044406414031982, 0.8148446083068848, -2.686206579208374, 1.8839744329452515, 1.1266039609909058, -0.6832161545753479, 0.7491253018379211, 0.7343569397926331, 0.8677257299423218, -0.19913743436336517, 1.3527861833572388, 2.2361531257629395, -1.5766096115112305, -0.9587054252624512, 0.015443389303982258, 0.37097105383872986, 1.604381799697876, 0.8932851552963257, 0.7081149816513062, -0.13120898604393005, -0.6606324315071106, -2.098663091659546, 1.0131460428237915, -0.3807131350040436, -1.1063199043273926, -0.520491898059845, -0.13978911936283112, -0.33557283878326416, 0.6362607479095459, 0.018645813688635826, 0.04707295447587967, -0.7873575687408447, 1.9756320714950562, -1.5926722288131714, -0.493106871843338, 0.8573393225669861, 0.5570217370986938, 0.4917062520980835, -0.14328822493553162, -0.5734130144119263, 0.3265892565250397, 0.6444730162620544, -0.4748552441596985, -1.7716484069824219, -0.26413512229919434, -0.7237221002578735, -0.38577598333358765, 0.09238273650407791, 0.5305579304695129, -0.7645575404167175, 1.1086138486862183, -1.5094127655029297, -1.4111241102218628, 1.0974094867706299, -0.2830698788166046, 0.41551336646080017, -0.620347797870636, 0.13351599872112274, 0.6496784090995789, -1.266146183013916, 0.612041175365448, 0.8850595355033875, -0.1054692268371582, 1.295561671257019, -0.9535450339317322, 1.079984426498413, 0.12917111814022064, 0.7136306762695312, 0.943308413028717, -0.07533007860183716, -0.8706541657447815, -0.3646191954612732, -0.3809720277786255, 0.749676525592804, -0.4635075628757477, -0.19016093015670776, 0.04244183376431465, 0.9696393609046936, -1.1614229679107666, -0.7415966391563416, 0.6902946829795837, 0.5566742420196533, 0.6792361736297607, 1.0546367168426514, 1.2471259832382202, 0.6533102989196777, -0.047940660268068314, 1.1958211660385132, -1.3399969339370728, 0.8260418772697449, -0.848323404788971, 1.173836588859558, 0.5898769497871399, -0.4653553366661072, 1.4125299453735352, 1.0950807332992554, 0.1630285233259201, -0.9516729712486267, -0.3974250853061676, -0.4850671589374542, -0.9701230525970459, 0.05309249088168144, 0.15146002173423767, -0.6272270083427429, 0.8118509650230408, 0.08633282035589218, 0.8799527287483215, -0.3720996379852295, -0.4979173243045807, 0.05501357838511467, 0.937085747718811, -1.072501540184021, -0.0018340791575610638, 0.8130327463150024, -0.8735329508781433, -0.883953332901001, -1.0352081060409546, 2.8102104663848877, -0.16940546035766602, 1.2441635131835938, 0.7652792930603027, -1.5003434419631958, -0.3039427697658539, 2.1335713863372803, -1.0764720439910889, -0.9557642340660095, -1.533979892730713, -0.2368514984846115, 2.2968828678131104, -0.6547879576683044, 0.3487681448459625, -1.1487853527069092, -0.5158751010894775, -0.3786737620830536, 0.46412327885627747, 1.7941505908966064, 0.6572672724723816, 1.3219014406204224, 0.5306840538978577, -0.32386183738708496, 0.8735321164131165, -0.08148377388715744, 0.297504186630249, -0.7976531386375427, 1.4490023851394653, -1.5232704877853394, 1.4356603622436523, -0.11226265877485275, 0.7647769451141357, -1.4292298555374146, 0.9674022197723389, 0.5478597283363342, -0.3969600200653076, -0.5589457154273987, -1.3597100973129272, -1.941622018814087, 0.17924292385578156, -1.266636610031128, -0.1075996458530426, -0.3247802257537842, 0.17299310863018036, -0.8578630089759827, 0.5265223979949951, 0.4074714183807373, 0.6594402194023132, 0.8338911533355713, -0.439319372177124, 0.17580637335777283, 1.3465687036514282, -0.30106621980667114, 0.05857788398861885, 0.13928838074207306, -0.781926155090332, -0.7244279384613037, 0.1042487621307373, 1.122727870941162, 0.16744717955589294, -0.5160247087478638, -0.8656877875328064, -0.40630102157592773, 0.2021084427833557, -0.5879935026168823, 0.7535504698753357, 0.45097577571868896, 0.23947766423225403, 0.8305506706237793, -0.84471195936203, -2.1625020503997803, 1.2952312231063843, 0.8826169371604919, 0.9517734050750732, 0.1584349125623703, 1.5155904293060303, -0.41963914036750793, -1.1817070245742798, -0.6022515296936035, 0.7647808194160461, -0.3520433306694031, 0.8040313720703125, 0.6960887312889099, 1.1133135557174683, -0.6349630355834961, 0.0652167797088623, -1.4147416353225708, 0.9245033860206604, -1.16492760181427, -0.298310786485672, 0.27677786350250244, 1.021752119064331, 0.5909900069236755, 1.1051902770996094, 1.299206018447876, 2.1431310176849365, -1.6513950824737549, -0.2094399631023407, 1.1629596948623657, -1.224703073501587, -0.2390143722295761, -1.3678655624389648, 1.1926697492599487, 0.6706494688987732, -0.4666735827922821, -3.0166313648223877, 1.0246891975402832, 0.6303417682647705, -0.6889968514442444, -1.521333932876587, -1.8932948112487793, 0.19633018970489502, 0.12791377305984497, 0.6960146427154541, 0.7208240032196045, -1.9148515462875366, -0.3366153836250305, 1.273918867111206, -0.8000735640525818, 0.6344402432441711, -0.4998030662536621, 2.3812248706817627, 1.2110790014266968, -1.4816330671310425, 0.6996808648109436, 1.254686713218689, -1.6786391735076904, 1.626325011253357, -0.2539829611778259, 1.4013992547988892, -0.07976201176643372, 0.6584463715553284, 0.2706625461578369, -0.04443996027112007, -0.1301344633102417, -0.839253306388855, -2.6159956455230713, -0.14341704547405243, 0.8053349256515503, -1.0585970878601074, -0.02452334389090538, -0.3052198588848114, -0.530145525932312, 1.774596929550171, 0.7675166726112366, -2.3429665565490723, 0.08649790287017822, 0.9220696687698364, 1.1243268251419067, -1.0797407627105713, 0.9485524892807007, 1.2160680294036865, 0.8353188633918762, 1.6143709421157837, 0.6474260687828064, -0.02174079604446888, -1.4218711853027344, 0.40837568044662476, 1.0612081289291382, 1.1792564392089844, 0.2730940878391266, 0.5841740369796753, -0.6515268087387085, 0.4715900421142578, -0.5683783292770386, -1.1750094890594482, 0.15928775072097778, -0.726921796798706, -0.10621261596679688, 1.3830081224441528, -0.5088456869125366, -1.4059520959854126, -0.8272382616996765, 0.46614786982536316, -1.2296240329742432, 0.228922039270401, 1.8006138801574707, -1.087384819984436, 0.6047384738922119, -0.6590767502784729, -0.44516125321388245, -1.5541439056396484, 0.9798324108123779, 1.5641430616378784, 1.2425928115844727, -1.9168808460235596, -0.5254802107810974, 0.19285686314105988, 1.569338321685791, -1.223691463470459, -0.7251693606376648, -1.1147578954696655, -0.09201354533433914, 0.5939873456954956, 0.20245477557182312, 1.0561871528625488, -1.6234368085861206, -0.7757126688957214, -0.9442191123962402, -0.4391289949417114, -0.6369847655296326, 1.0437244176864624, 0.017351482063531876, -0.6060698628425598, 1.2064517736434937, 0.733390748500824, -2.1671688556671143, -0.9274616241455078, 0.20655196905136108, -0.5180473923683167, -0.6768784523010254, -0.4002342224121094, 0.46182042360305786, -2.549862861633301, -1.2460336685180664, -0.6116573214530945, -1.6957522630691528, -0.5978540182113647, -1.2096325159072876, 0.3620452880859375, 1.3263723850250244, 1.1452816724777222, -0.10531637817621231, -0.6696611642837524, -0.08212372660636902, -2.617852210998535, -0.4975544214248657, -0.7481773495674133, -0.38993075489997864, 0.506815493106842, 2.442445993423462, -0.42929619550704956, 0.8091009259223938, -0.35457420349121094, 0.757197380065918, 0.3607003092765808, 0.5696064233779907, -0.006126405205577612, -0.7446461319923401, 0.17582015693187714, 1.205380916595459, -0.7625685930252075, 1.4135137796401978, -1.0956740379333496, 0.46139517426490784, 0.7405052781105042, 0.5376048684120178, 1.37047278881073, 0.6177851557731628, -0.020888566970825195, 0.31364327669143677, 0.427034854888916, -0.371831476688385, 0.3709140121936798, -0.7808641195297241, 1.6742198467254639, -0.36871516704559326, 0.28966906666755676, -1.1345669031143188, -0.4591914415359497, -0.4197846055030823, -0.4959515929222107, 0.1207653060555458, -0.6741845011711121, -1.6518614292144775, -0.8166680335998535, 0.8172792792320251, -0.6695972681045532, 0.6522033214569092, 1.4032294750213623, -0.42964816093444824, 0.9863392114639282, 1.9035853147506714, -0.771177351474762, 0.2840692102909088, -1.8077970743179321, -2.2096505165100098, 0.3233705461025238, -0.09573717415332794, -0.7430870532989502, -0.7487248182296753, -0.7771941423416138, 0.2537366449832916, 1.5880995988845825, -0.25783783197402954, 1.280290961265564, 0.2065800279378891, 0.43566590547561646, 0.4153324067592621, 0.06476932764053345, 1.352226734161377, -1.8454556465148926, -1.635764718055725, -0.1386422961950302, 0.30319899320602417, 2.1299729347229004, -0.7213200330734253, -0.22177940607070923, -0.2556411027908325, 2.0892393589019775, -0.5797481536865234, 0.3421412706375122, -1.4149417877197266, -1.5779703855514526, -1.8347214460372925, -0.8656993508338928, -0.005772534757852554, 2.00325083732605, 0.9302616119384766, 0.2839314639568329, -0.4429594874382019, 0.466069757938385, -1.1116399765014648, -1.5313310623168945, 0.9670793414115906, -2.3760111331939697, -0.10760856419801712, 0.3907463252544403, -0.1468215435743332, -0.1498635858297348, -0.0004580789536703378, 0.537420392036438, 1.2357124090194702, -0.4109613597393036, -0.09040405601263046, 0.1498866081237793, 0.39448219537734985, -0.6322929859161377, -0.5405246615409851, 0.6276388764381409, -1.6603137254714966, 0.5137055516242981, 1.8371917009353638, 0.7519255876541138, 0.5058389902114868, -0.23398180305957794, -0.3834288716316223, 0.283439576625824, 0.003188307862728834, 0.006387775763869286, 0.1979060024023056, 1.84538733959198, -0.2401888221502304, 1.3837382793426514, 0.4370104670524597, -2.151343584060669, 0.8590401411056519, -0.6632708311080933, 1.3115525245666504, 0.38042306900024414, 0.03217959403991699, -0.384768009185791, 0.5074734091758728, 0.07638148963451385, -1.2605799436569214, -1.4681493043899536, -0.7603335380554199, 1.2729512453079224, 0.8930479288101196, 1.4577161073684692, -0.8585364818572998, -0.38792693614959717, -0.8614429235458374, 0.5243949294090271, -0.684576690196991, -1.4224075078964233, -0.07072649896144867, -0.24910660088062286, -0.49812406301498413, 2.2664740085601807, -1.1115214824676514, 0.3802013695240021, -1.5811634063720703, -1.1890538930892944, -0.540752112865448, 0.44195249676704407, -0.7312541007995605, 0.3494867980480194, -0.5281299352645874, -0.27502068877220154, 0.6796510815620422, -0.8265323042869568, 1.0771424770355225, 0.34642234444618225, 0.036994002759456635, 0.17186938226222992, -2.522148847579956, -0.9349673390388489, -0.8042489886283875, -0.43472057580947876, -1.1717346906661987, 0.2453027367591858, -1.1709345579147339, -1.6867969036102295, -0.9991939067840576, -3.0140469074249268, -0.3034536838531494, -0.20919300615787506, 0.5922977924346924, -0.38175326585769653, 0.006693442352116108, 0.19195984303951263, 1.751477837562561, -0.018529310822486877, -0.8034170866012573, -0.5789446234703064, -0.2240968495607376, 1.4315452575683594, -0.4485386610031128, -1.1471312046051025, -0.6843093037605286, -1.5465481281280518, -0.22128397226333618, 0.10058016330003738, 0.23058003187179565, 1.4061686992645264, -0.39914315938949585, 0.8078485131263733, -0.21626922488212585, 0.88245689868927, -0.2706509232521057, -1.4477949142456055, -0.05042227357625961, 1.923773169517517, -1.8822834491729736, 1.047448992729187, -0.0607505738735199, 1.1752209663391113, 0.7508646845817566, -0.12354431301355362, 1.2095541954040527, 0.4514176547527313, 0.5153706669807434, -3.000807285308838, -1.8921838998794556, 0.038239482790231705, -0.3300688564777374, -0.38578733801841736, 0.07576233893632889, -0.8331488370895386, 1.5704971551895142, -1.0616153478622437, -1.0819430351257324, 1.5797877311706543, 0.49647971987724304, -0.5135146379470825, -1.302296757698059, 1.823277235031128, 0.4361288547515869, 0.5891921520233154, 2.4745755195617676, 0.007983026094734669, -0.2000856250524521, 0.004785777069628239, -0.060269687324762344, 1.0396450757980347, -1.8211541175842285, 0.399053156375885, -1.406185269355774, 0.02222561649978161, -0.4065914750099182, 0.2530471086502075, -0.020205529406666756, -0.2167554795742035, 0.2489597648382187, 0.8745629191398621, 0.7266759872436523, 0.38527771830558777, 0.2389134168624878, -1.1574667692184448, 1.6471182107925415, -0.2766750454902649, 0.0699758529663086, -0.2118355929851532, 0.7327378392219543, -0.9260236620903015, -1.1246200799942017, 0.5336470603942871, 0.583176851272583, -0.206231951713562, -0.22480346262454987, -1.1211576461791992, 0.8226466178894043, 0.15523797273635864, -1.3905178308486938, -0.18932053446769714, 1.1585662364959717, -0.6661419868469238, 0.909016489982605, 0.7402642965316772, -0.9515727758407593, 0.008546230383217335, 0.1259918063879013, 0.46741777658462524, -0.2553994357585907, 1.7240310907363892, 0.892466127872467, 0.42027920484542847, 0.19572176039218903, -0.5737081170082092, 0.9621630907058716, 1.3373212814331055, 2.0941965579986572, 1.9570165872573853, -1.847780704498291, 0.6080387830734253, 0.17940449714660645, -0.8814759850502014, -1.8372151851654053, 0.9860157370567322, -1.7479772567749023, 0.5160592198371887, 1.5757204294204712, -1.162387728691101, -1.0965937376022339, 1.2306723594665527, 0.04731196165084839, -0.014295953325927258, -0.828404426574707, 0.06250910460948944, 0.6580788493156433, 0.31893959641456604, -0.12091069668531418, 0.5533717274665833, -1.313423991203308, 0.35724762082099915, -0.4218333661556244, -0.27041780948638916, 0.22630123794078827, 1.9533731937408447, 0.46735766530036926, -0.34140709042549133, -0.6055867075920105, 1.2149755954742432, 2.5620946884155273, -0.5481942892074585, -0.545871913433075, -0.5142483711242676, 1.2782584428787231, -0.7391573786735535, 2.5508320331573486, -0.25565528869628906, -2.144174575805664, 1.8502187728881836, 0.17707014083862305, 0.5566261410713196, -1.561442255973816, 0.4342566728591919, 0.010455389507114887, 0.987027645111084, 0.3068474531173706, 0.1448196917772293, -0.20998455584049225, -1.0709455013275146, -0.6318659782409668, 1.7796252965927124, 0.06643784046173096, -1.490168571472168, 0.33974310755729675, 0.5202921032905579, 0.1895385980606079, -0.6539986729621887, -0.2830185890197754, -0.9309921264648438, -1.3279393911361694, -0.09248914569616318, 0.0912182480096817, 0.7825270295143127, 1.507073998451233, 0.7238785028457642, 1.5650078058242798, -0.3467504680156708, -0.6547819972038269, 0.09562789648771286, -0.6638118028640747, 0.10117659717798233, 0.9872515797615051, -0.33469638228416443, 0.7502966523170471, -0.9326404333114624, 1.0111004114151, 0.4709188640117645, -0.7958707809448242, -0.37262678146362305, -1.6713941097259521, -2.584801435470581, 0.47495773434638977, -0.057385899126529694, 0.32317620515823364, -0.06522880494594574, 0.8339911103248596, 0.05681027099490166, -0.06272783130407333, -2.2011353969573975, 0.5321580171585083, -1.1464437246322632, -0.27137261629104614, -0.8733079433441162, -0.705696165561676, 1.412532925605774, 1.1097333431243896, -0.24267378449440002, -0.18081136047840118, 0.6469231247901917, 0.4979613423347473, -1.2278515100479126, 0.9550405740737915, 0.6006579399108887, -1.1826026439666748, -0.08941567689180374, -0.09009213000535965, -1.879667043685913, 0.16825252771377563, -0.21055898070335388, 1.7915623188018799, 0.48895037174224854, -1.033281683921814, 0.3997550308704376, 1.0476723909378052, -1.7346694469451904, 1.1444764137268066, 1.7078250646591187, -0.6850731372833252, -1.5252526998519897, 1.3273186683654785, -1.098749041557312, -0.8318929672241211, 1.1318542957305908, -0.31201982498168945, -0.592051088809967, 0.4073905944824219, 0.3449307680130005, 0.9226457476615906, 1.3679083585739136, -1.3712317943572998, -0.4530404508113861, -0.053501687943935394, -0.9501909613609314, -1.1169965267181396, 0.9291352033615112, 1.5876606702804565, -1.0188097953796387, 0.40260472893714905, 2.1943957805633545, -1.5095523595809937, -0.2042720913887024, -1.7261130809783936, -1.0877996683120728, 1.3151885271072388, 0.24968940019607544, 0.20173460245132446, 0.7874842882156372, 0.22226227819919586, -0.6197872757911682, 0.21505697071552277, -0.4422307014465332, -0.007352746091783047, 0.12937811017036438, 1.1893314123153687, -1.307753562927246, 0.4295322299003601, -0.43962758779525757, -1.260176420211792, -0.03841901570558548, 0.037020936608314514, -0.6505497097969055, -1.1007566452026367, 0.8433857560157776, -0.0971837267279625, -0.08449570089578629, -0.17044954001903534, -1.1367526054382324, -0.416756808757782, -0.2128620594739914, -0.14504392445087433, -2.129340171813965, 0.4693625867366791, 0.06929393857717514, -0.024941591545939445, -0.35214975476264954, 1.1775809526443481, 1.3131294250488281, 0.33491599559783936, -0.12661541998386383, 0.14331625401973724, 1.1792861223220825, -0.8663002848625183, -1.6826814413070679, 0.43490976095199585, -0.7319344282150269, -1.2129942178726196, 0.7070540189743042, 0.5428631901741028, 2.9990670680999756, 0.480663925409317, -0.32943132519721985, -1.621483564376831, -0.44202303886413574, 0.5818076729774475, 0.6840358972549438], [0.3282131254673004, 1.700578212738037, -0.1694224774837494, -0.11105965822935104, 0.4664176404476166, -0.2892020344734192, -0.039326250553131104, 1.6143677234649658, 0.5960620045661926, 0.011291726492345333, -0.5807839035987854, -0.7961060404777527, 1.2335104942321777, -1.2390307188034058, 1.0156720876693726, -1.6246907711029053, -0.09282655268907547, 0.7928259968757629, -1.0672473907470703, -1.0696343183517456, 0.392973393201828, 0.3991923928260803, 1.5763521194458008, 3.3254613876342773, 0.2506418526172638, 2.0858449935913086, -1.3374881744384766, 0.8848941326141357, 0.03769606724381447, -0.6829758286476135, -1.3050556182861328, 0.4443776607513428, -1.560557246208191, -0.555020272731781, 0.2980212867259979, 0.9301402568817139, -0.7668992877006531, -0.3761657476425171, 1.6873886585235596, -0.852989673614502, -0.040734488517045975, -0.8474371433258057, 1.0660828351974487, -0.3517543077468872, 0.0074519263580441475, 8.384051761822775e-05, 0.4145794212818146, -0.7978172302246094, -1.1864138841629028, -0.379305362701416, 0.564045250415802, -1.2983509302139282, 0.17006000876426697, -0.9735199809074402, -0.02190711349248886, 1.7630672454833984, 1.292620301246643, -0.5435222387313843, -0.9027212262153625, -0.9235066175460815, 1.4440261125564575, -1.0171866416931152, 1.5300180912017822, 1.0191117525100708, 0.6425408124923706, -0.052578993141651154, -1.4464541673660278, -2.5435283184051514, 0.881162703037262, -0.13218332827091217, -1.6416749954223633, 1.1230272054672241, -0.44375917315483093, -0.4737487733364105, 0.2536959648132324, 0.8921762108802795, 0.6879485249519348, 0.8153063654899597, -1.2093771696090698, -0.06338739395141602, -1.172155737876892, 1.6882025003433228, 0.0025010108947753906, -0.5487344264984131, -0.20096322894096375, 0.5904308557510376, 0.35647138953208923, -0.4492739140987396, 1.2188185453414917, 0.3336981236934662, 1.181937336921692, -0.7521777749061584, -0.2701394259929657, -1.473354697227478, -0.3230469822883606, -0.4252481758594513, 0.5645111203193665, 0.6714119911193848, 0.34812912344932556, 0.10822165012359619, 1.3003371953964233, 0.6878430843353271, 0.7844003438949585, 0.8019979596138, -0.6111853122711182, -0.1365450918674469, -0.03896569460630417, 0.3294732868671417, 0.02084992080926895, -0.40273404121398926, 0.19386348128318787, 0.538358211517334, -1.6504138708114624, -1.028375267982483, 0.10781285911798477, -0.3064934015274048, 0.18792882561683655, -0.2599932849407196, 0.40944817662239075, -1.4016062021255493, 0.33406662940979004, 0.8147111535072327, -0.7292548418045044, -0.6514568328857422, 0.5911462306976318, 0.7948952317237854, 0.8132086396217346, 1.995829463005066, 1.2245012521743774, 2.1771509647369385, -1.5622810125350952, -0.24079059064388275, -0.2510596811771393, 2.1340816020965576, 0.7988190650939941, 1.2827988862991333, -0.20552097260951996, 1.9146323204040527, -0.6945475935935974, 0.86823970079422, -1.2468020915985107, 0.698884904384613, -2.3520264625549316, 1.089969515800476, -0.8437560796737671, -0.6453250050544739, -1.0421090126037598, 1.0669910907745361, 0.5898032784461975, 0.981080949306488, -0.6608591079711914, 0.41633179783821106, -0.08725976943969727, 0.6481121778488159, -0.23119300603866577, -0.2721478343009949, -1.9510384798049927, 1.0639722347259521, 0.641433835029602, 1.2021887302398682, -0.2785710394382477, 1.1928575038909912, 0.43646079301834106, 0.09340224415063858, -0.6605132222175598, -0.38956382870674133, -0.5245075225830078, 0.4796188771724701, 0.037807974964380264, -1.9265340566635132, 0.06302820146083832, -0.035295527428388596, 0.46099039912223816, 0.47972866892814636, -0.2726035416126251, 1.1899222135543823, -0.3068641424179077, -2.3525445461273193, 0.5483470559120178, -0.4796638786792755, -0.3626335859298706, 1.1291793584823608, -0.5753381848335266, 1.3774646520614624, -2.017789363861084, 0.5463285446166992, -1.249996542930603, 0.6028389930725098, -1.170925498008728, 1.2450779676437378, -0.5281623005867004, -1.1925344467163086, -1.3436113595962524, -0.3679410219192505, 0.9214332699775696, 0.37703779339790344, 1.171778678894043, 0.024248773232102394, 1.0851538181304932, 0.761369526386261, 0.6501603126525879, -2.4016032218933105, 0.3100075125694275, -0.2255559116601944, 0.3135707378387451, 1.5675157308578491, 1.1568983793258667, 0.7431592345237732, 0.5616354942321777, 0.471731573343277, 0.5539547801017761, 0.7336273789405823, -0.5868513584136963, 1.6992554664611816, -0.27740979194641113, -1.249178409576416, -0.9001739025115967, 0.5619314312934875, 0.07144099473953247, 0.4704757332801819, -0.14016437530517578, -1.7980718612670898, 1.1578917503356934, -0.9054322838783264, -0.9055839776992798, -0.8817362785339355, -0.2773623764514923, -0.7979410886764526, -1.41892409324646, -0.5244202613830566, -1.1463168859481812, -1.967397689819336, 1.1626198291778564, 0.3865652084350586, -0.9789910316467285, 1.1002662181854248, 1.3779217004776, -0.36479106545448303, -0.47395291924476624, 1.184380292892456, 2.005990505218506, -0.867470920085907, -0.015697119757533073, 1.8989918231964111, -1.3546149730682373, 0.29618874192237854, -0.35887032747268677, -0.7520700693130493, -0.6811544299125671, 0.17106308043003082, -1.4043045043945312, -1.1174671649932861, -1.2326252460479736, 1.0486571788787842, 0.027319306507706642, -2.4777138233184814, -0.7989462614059448, 0.30353888869285583, -0.48630964756011963, 1.1175702810287476, -0.850771427154541, -0.48042377829551697, -1.5418421030044556, -2.0074408054351807, 2.2764341831207275, -0.2199964076280594, -1.2674295902252197, 0.8554463982582092, 0.44856956601142883, 2.2540438175201416, -1.980887770652771, 0.8827662467956543, 0.203607439994812, 0.9881712794303894, -1.1055247783660889, -0.18748871982097626, 0.40400969982147217, -0.1747937798500061, -0.4112998843193054, 1.5284663438796997, 1.0830971002578735, -0.9516521692276001, 0.8111305832862854, 0.7558593153953552, 0.0458117239177227, -1.0382829904556274, 0.23544242978096008, -0.2289949357509613, 1.9350509643554688, -0.2424628734588623, 0.22969834506511688, -0.48279106616973877, -0.30039748549461365, -0.24152956902980804, 1.544463038444519, -0.6764381527900696, 0.10518334805965424, 0.9350113868713379, 1.21605384349823, 0.2712087631225586, 1.0237343311309814, 0.49174579977989197, -0.26701661944389343, -0.8527016639709473, 0.652000367641449, -1.465673565864563, -1.656104564666748, -0.9561894536018372, -1.0055996179580688, 2.3664028644561768, 2.190544605255127, 0.34003710746765137, 0.013690445572137833, -0.3141290843486786, -0.04555730149149895, -0.5865045189857483, -0.8757193088531494, -0.4950833320617676, 0.17929252982139587, 0.7746994495391846, -0.4138703942298889, -0.5019845962524414, -0.46083709597587585, 0.5440505743026733, -0.2926754653453827, -0.22897927463054657, -0.3429122567176819, -0.4617004990577698, 0.46387019753456116, -0.24425159394741058, 0.9919430017471313, 0.7763047814369202, -0.6019619703292847, 0.1086263358592987, 1.804214596748352, 0.4665268659591675, -1.856848955154419, -0.17877796292304993, 0.9535209536552429, 0.14160753786563873, 0.1935870498418808, -0.19026488065719604, 0.31431788206100464, 0.31476420164108276, 0.6220808625221252, -0.14008623361587524, 1.386999249458313, -0.4862520396709442, 1.3511000871658325, 0.07067429274320602, 0.9312432408332825, -1.5270270109176636, -0.677617073059082, 0.6082412004470825, 1.402083396911621, -0.8700440526008606, 0.13579535484313965, -0.45116081833839417, -1.399786353111267, 1.5793811082839966, -1.270426869392395, 1.624207854270935, -1.694995403289795, 0.4597371816635132, 0.4407399296760559, 0.019343405961990356, -0.979193925857544, -1.0001389980316162, 0.782953679561615, 1.5980224609375, 0.06480993330478668, -3.0016744136810303, -0.19739335775375366, 1.3332107067108154, -1.1323919296264648, -0.28256916999816895, -0.9241300821304321, -0.722878634929657, -0.2642381191253662, 0.01592741347849369, 1.1394249200820923, -0.09264905750751495, 0.5463253855705261, -1.0185185670852661, 1.994608759880066, -1.5820014476776123, -0.6050653457641602, -0.4980805516242981, 0.00029430584982037544, 0.2902914583683014, 1.0568605661392212, -1.2369821071624756, 2.104748487472534, 0.8561425805091858, -0.12969741225242615, -0.3715144395828247, -2.9232559204101562, 0.41262587904930115, 1.497390627861023, 0.4020053446292877, 1.808472752571106, 0.4860686659812927, 0.542151927947998, 2.2475192546844482, -0.6410690546035767, 0.2856232821941376, -0.48908570408821106, 1.3706843852996826, 0.8271484375, 1.1789227724075317, 0.03892487660050392, 0.053155671805143356, -0.30170512199401855, 2.171426296234131, 0.07630948722362518, 0.8291304707527161, -2.231088399887085, 0.4602273106575012, 0.6021113395690918, 1.0274287462234497, 1.1370809078216553, -0.5061720609664917, -0.18644636869430542, 0.3831135630607605, 1.5896551609039307, 0.885994553565979, 0.05050182342529297, -0.6751652359962463, -1.6732946634292603, 0.8999554514884949, 0.872641384601593, 0.9662365317344666, -0.678691029548645, 0.4854509234428406, 0.9911715984344482, -0.7544312477111816, -2.4389355182647705, 0.6655257344245911, 0.8840669393539429, 0.7684952020645142, 1.9588820934295654, -0.0923520177602768, -1.4556076526641846, -1.4915589094161987, -0.6587094664573669, 0.10899396240711212, 0.2049531787633896, 0.16627688705921173, 0.5461510419845581, 0.014566527679562569, -2.920870065689087, -1.0320943593978882, -0.5738005042076111, 0.8283360600471497, -0.21609878540039062, -0.08143561333417892, 1.1261236667633057, 1.8871599435806274, 0.21155665814876556, 1.5280309915542603, -0.4192444086074829, 0.6691301465034485, 0.0764133632183075, 0.38377660512924194, 1.536789059638977, 0.9942957758903503, -1.443371295928955, -2.0651822090148926, 1.0603671073913574, -1.2114291191101074, -0.9986497163772583, -1.809951663017273, 1.599483609199524, -1.1554259061813354, 0.6570937633514404, 0.5454612374305725, 0.16403743624687195, 0.6375913619995117, -1.4643354415893555, -0.7931070327758789, 0.5794129967689514, 0.12784691154956818, 0.08573548495769501, 1.529013752937317, -0.24796049296855927, 0.2983631491661072, -0.5035758018493652, -1.014636516571045, -0.8552213907241821, 1.0840067863464355, 0.15534038841724396, -0.6525525450706482, 1.1494274139404297, 0.8945828080177307, -0.8393915891647339, 1.0168330669403076, -1.7936815023422241, -0.18797370791435242, -0.33798402547836304, 1.9339780807495117, -0.37156960368156433, 1.2359620332717896, -0.6520262956619263, 0.19795364141464233, -0.11583776772022247, 0.4644199013710022, -0.16796772181987762, 2.7757437229156494, -1.2477450370788574, -0.20572803914546967, 1.5161024332046509, 0.6121892333030701, -0.8213573098182678, 0.4022693932056427, -1.3995134830474854, 1.4984358549118042, -0.9521977305412292, -0.5618877410888672, 0.6472955346107483, 0.0006849250057712197, -1.7293217182159424, -2.325822114944458, -1.0734689235687256, -0.513902485370636, -0.9100086688995361, 0.7985571622848511, 0.6395692825317383, -0.40832674503326416, 0.5174299478530884, -1.7095013856887817, -0.01915503293275833, -0.34133192896842957, 0.8215474486351013, -0.4067036509513855, 0.31657537817955017, -0.7829869985580444, -0.9997904300689697, -1.2662485837936401, -2.2110440731048584, -0.8160882592201233, -0.7830075621604919, 2.592546224594116, -0.2794789671897888, 0.13576531410217285, 0.07987410575151443, 1.8058103322982788, 1.0084928274154663, -1.2920100688934326, -0.17647449672222137, 1.4118291139602661, 1.44113290309906, 1.374645471572876, 0.8111358880996704, 0.4461286962032318, 1.161944031715393, 0.38058769702911377, -0.5188397765159607, -0.3543117046356201, 0.9355397820472717, 1.8069268465042114, -0.9227965474128723, 1.533967137336731, -0.44672757387161255, -0.3336661458015442, 0.7197282910346985, 0.09691175073385239, 0.28717538714408875, 0.8161963820457458, -0.6911816000938416, -0.15061655640602112, -0.14614196121692657, -0.7914361953735352, 0.35861971974372864, -0.846434473991394, 0.6584359407424927, -0.3260557949542999, 1.2073540687561035, -1.4468281269073486, -1.3236885070800781, 2.468369483947754, -1.2522640228271484, 0.2544364333152771, 0.038442131131887436, -1.6374902725219727, -0.5933089256286621, -0.8933212161064148, -0.45832589268684387, 2.391488790512085, 0.12285617738962173, -1.225562572479248, 1.1075739860534668, -0.6370603442192078, 1.7319823503494263, 2.410320997238159, 0.06170656159520149, -0.8726546764373779, 1.5747814178466797, -0.37853842973709106, 1.3894203901290894, -0.5302879214286804, -0.12408660352230072, 1.5911773443222046, 1.033840537071228, -1.4548901319503784, -0.8263914585113525, -0.8206437230110168, 1.9687381982803345, -1.2619907855987549, -0.7473974227905273, 0.10864745080471039, 1.401959776878357, -1.3483730554580688, -0.44092950224876404, -0.6800618767738342, -0.11309529095888138, -1.068216323852539, -1.6378165483474731, -0.15258526802062988, -0.6305074691772461, -1.2240427732467651, -0.2061210423707962, 0.32749906182289124, 0.9182238578796387, 0.3508416712284088, -0.33358636498451233, 0.33391866087913513, 1.1217972040176392, -1.3637059926986694, 1.0192121267318726, -0.2321658432483673, 1.243250846862793, 0.18209943175315857, -0.8255894780158997, 0.9258149266242981, -1.1115280389785767, 0.7927206158638, 1.391624927520752, -1.0238157510757446, 0.0358772873878479, -1.5692507028579712, 0.19791966676712036, -1.0701515674591064, -0.8483311533927917, 0.9387752413749695, -0.37522202730178833, 0.5925488471984863, -0.6386644840240479, -0.04836929962038994, -0.015517971478402615, 1.7207213640213013, 0.855087161064148, -0.8173906207084656, 1.5115008354187012, -0.7477481961250305, -0.15739305317401886, 1.1561580896377563, -0.4631674587726593, 0.03769339248538017, 0.5748910903930664, 0.3795468807220459, -1.717863917350769, 1.3798718452453613, 1.4440406560897827, 1.4357341527938843, 0.5691208839416504, 1.4300047159194946, 1.1436080932617188, -0.7744152545928955, 1.1978408098220825, 1.071542739868164, 1.8528294563293457, -0.01744610071182251, 1.000270962715149, -1.2648975849151611, -0.07769085466861725, 0.2899320423603058, -1.32230806350708, 0.3026595413684845, 0.8810376524925232, -0.5122841596603394, -0.25498127937316895, -0.33122262358665466, 0.12043636292219162, -1.6837471723556519, 0.3375554382801056, 0.13965871930122375, -0.4889194667339325, 0.09734444320201874, -0.9788679480552673, 1.4857444763183594, 0.9439864158630371, 1.6983438730239868, -0.7656742930412292, -0.21706631779670715, -0.8884174823760986, 0.18690215051174164, -0.10437680035829544, -1.8359930515289307, -0.7557465434074402, 1.7664443254470825, -0.4733479619026184, -0.7689529061317444, -1.0241763591766357, -0.6288099884986877, 0.6552938222885132, 0.030578602105379105, 0.16279752552509308, 0.6352896094322205, -0.7705678343772888, 0.2534794807434082, 1.1147887706756592, 0.6913265585899353, -0.06345571577548981, -0.26805976033210754, 0.30167022347450256, 0.45919865369796753, -0.535226047039032, 1.408132553100586, -0.4745694100856781, -1.653177261352539, -0.7128162384033203, 0.5960143804550171, -1.489063024520874, 0.17600542306900024, -1.3122577667236328, -1.6871917247772217, -1.8959968090057373, -1.0909146070480347, -0.7630619406700134, 0.2903233766555786, -0.6818579435348511, 1.5859547853469849, -0.4698225259780884, 0.3287135064601898, -0.5679869055747986, 0.4069872796535492, -0.33735328912734985, -0.21229492127895355, 1.1234134435653687, 0.7138932943344116, 0.9359742403030396, 0.9611260294914246, 0.1806531399488449, -0.3563154935836792, -0.44964131712913513, -1.6768122911453247, -0.10103294998407364, -0.3691834807395935, -1.0474252700805664, -1.5089167356491089, 0.9240144491195679, -0.23474131524562836, -0.5897151827812195, -0.14372816681861877, -0.009848637506365776, -1.03818941116333, -1.165056586265564, 1.163055419921875, -0.05125223845243454, 1.354223370552063, 0.14473672211170197, 0.39727136492729187, -0.9248589277267456, -0.10743309557437897, -1.7146300077438354, 1.466561198234558, -1.3599013090133667, -0.32133328914642334, -0.8799957633018494, -0.009540234692394733, -1.5771411657333374, 0.8385875821113586, -1.467971920967102, 0.22521425783634186, 0.6312170624732971, 0.8811239004135132, 0.7091163396835327, -1.064711570739746, -0.01268645841628313, 0.4129408597946167, -0.030617259442806244, -0.49680590629577637, 0.49944862723350525, 1.0357939004898071, 0.33241114020347595, -2.343787431716919, 1.5833669900894165, 0.5977826714515686, -1.030799388885498, 0.17526979744434357, 0.2691139280796051, 0.5903995037078857, 0.44718605279922485, 0.20076745748519897, 0.32417285442352295, 0.5636249780654907, 0.8703179955482483, -0.9951255321502686, -0.4918297827243805, 0.9034394025802612, -1.6649361848831177, 1.0492610931396484, 1.031821846961975, -1.4322813749313354, 1.5311716794967651, -0.7084299921989441, 0.39272940158843994, 1.4578109979629517, -0.7799923419952393, -1.9810141324996948, 0.1835661083459854, -0.5981374382972717, 1.813317060470581, -0.7288573384284973, -0.34413835406303406, -0.10055116564035416, 0.8820255994796753, -0.6454129219055176, -0.7229374051094055, -0.7956246733665466, 1.9118316173553467, 0.8460931181907654, -0.2644117474555969, 0.2522004246711731, 0.4427160918712616, 0.04717271775007248, -0.688083291053772, -1.4956116676330566, -0.016491685062646866, -0.3078118860721588, -0.4672173857688904, 0.1950463205575943, 0.16026775538921356, -0.9215518832206726, 1.5396504402160645, -0.544244647026062, -0.12286076694726944, -1.968002200126648, -0.8909217119216919, 0.5007671117782593, 1.1281805038452148, -1.270717978477478, -0.12842248380184174, 0.6086742281913757, 1.1782431602478027, 0.9327571988105774, -0.5499173998832703, 2.4618213176727295, -0.2215259075164795, -0.7800171375274658, 0.070714570581913, -0.2223145067691803, -0.35726258158683777, 0.9466732144355774, -0.18901170790195465, -0.4239344298839569, 0.6890767216682434, -1.1557888984680176, 0.27934110164642334, -2.122485399246216, -0.32685720920562744, -0.4342830777168274, 0.5109108090400696, -0.4568198621273041, -1.3381284475326538, 0.1424107402563095, 0.9296207427978516, -0.16838160157203674, -0.35566282272338867, 0.8391913771629333, 0.4028104543685913, -1.5525239706039429, -1.8035662174224854, 0.5884115099906921, 0.028518961742520332, -0.7837536334991455, 0.4189715087413788, 0.44974595308303833, 0.7223452925682068, 2.6242964267730713, 0.018092798069119453, -0.5876688361167908, 0.17617551982402802, -0.028524626046419144, 0.6217324137687683, -0.6643955111503601, -0.4223778545856476, 0.0917612612247467, -0.14367738366127014, -1.1823029518127441, -0.7602264285087585, -0.7117805480957031, -0.5676447749137878, -0.5703603625297546, -0.4495372176170349, 0.6419972777366638, -1.2501980066299438, 0.13479359447956085, -1.7582476139068604, -3.2717342376708984, 1.2499781847000122, -0.38636302947998047, -0.31729742884635925, -0.9329755306243896, 1.7615854740142822, 0.6863068342208862, 0.43182194232940674, 1.2814664840698242, -1.9395198822021484, -0.524846613407135, -1.7557698488235474, 0.5800029039382935, 0.5001806616783142, 0.8249468207359314, -0.5920767188072205, -0.2585948407649994, -0.6038404107093811, 0.31037023663520813, 1.457983136177063, 1.1845049858093262, 0.4686097800731659, 0.507439136505127, 1.1708639860153198, -0.029461093246936798, -0.8543294072151184, 0.09679523855447769, -0.47193440794944763, 1.0770913362503052, -0.5472943782806396, -0.4961111843585968, 2.5551035404205322, 0.19388717412948608, 0.44247278571128845, 0.8389334678649902, 0.38279491662979126, -0.7945408225059509, -0.8138251304626465, -0.89699387550354, 0.367056667804718, 1.0472415685653687, -1.8699992895126343, 2.049943208694458, -0.8341339826583862, 1.1349912881851196, -0.9237644672393799, 0.4837687611579895, 1.853991150856018, -0.05451821908354759, 0.9299312233924866, -0.9684138298034668, 0.7551920413970947, -1.051364779472351, -0.5934375524520874, -0.2253929227590561, -0.9335340857505798, 1.057293176651001, -0.04122204706072807, 0.07351315021514893, 1.1084767580032349, -0.09093856811523438, 0.6576269865036011, -0.1196901872754097, 0.34077832102775574, 2.316056966781616, 0.9073043465614319, 0.44391879439353943, -2.01194429397583, -0.651441752910614, 2.040915012359619, -0.024875309318304062, -0.1985078752040863, -2.874903678894043, -0.6625097393989563, -0.3902161419391632, 0.31170010566711426, -0.2356959581375122, -0.02060537599027157, 0.7025781869888306, -0.5473119020462036, -0.34225839376449585, -0.2084963619709015, 0.8832873702049255, -0.7387139201164246, 0.4940745532512665, -1.731240153312683, -0.8423009514808655, -0.2676600515842438, 0.10470355302095413, -0.28308889269828796, -0.6047604084014893, 0.5730195045471191, -2.0434417724609375, 0.039097175002098083, -0.6203855872154236, -0.8512043952941895, -0.30098646879196167, 0.13066993653774261, 0.383819580078125, -0.0900525376200676, 2.2094919681549072, 0.3313659429550171, 1.2945767641067505, -0.010289023630321026, 0.5489491820335388, -2.0246739387512207, -0.08285392075777054, -0.24990755319595337, 0.7920053601264954, -0.11956169456243515, -0.855781078338623, -1.4559435844421387, -1.5630542039871216, -0.9873470664024353, 0.8011367321014404, -0.2083522230386734, -0.058075614273548126, -0.7752638459205627, 0.7153598070144653, 1.5740822553634644, 0.6394517421722412, 0.3278777301311493, 0.8853864669799805, 0.7164055705070496, -0.13694530725479126], [-0.2914671301841736, 0.37834158539772034, 0.7091058492660522, -0.8558598756790161, -1.433524250984192, -0.38457995653152466, 0.6453424096107483, -0.2565959095954895, -1.0932472944259644, -1.0623877048492432, 0.3578459322452545, -0.30271002650260925, 0.9174846410751343, -1.3451231718063354, 0.7085657119750977, -1.5664304494857788, 1.0536383390426636, -0.8581291437149048, 0.25293681025505066, 0.08993392437696457, 1.0725007057189941, -0.3234472870826721, -0.8855540752410889, -0.44578465819358826, 0.03198718652129173, -0.7600828409194946, -1.759667992591858, -1.3287498950958252, 1.0001579523086548, 0.435494065284729, 2.9603428840637207, -0.7599533200263977, -1.11277437210083, 0.1743813306093216, -0.30632495880126953, 0.18358077108860016, 0.04485902562737465, -1.2488631010055542, -0.06458096206188202, 1.893316388130188, -0.5621264576911926, 1.6814115047454834, 0.1172320544719696, -1.1951518058776855, 0.8420009016990662, 0.18031205236911774, -1.1847559213638306, 0.731467068195343, -1.604818344116211, -0.22245405614376068, 0.12310346215963364, -0.2736606001853943, -0.002317538484930992, 0.631953239440918, 1.422371506690979, 0.817721962928772, 0.5481870174407959, 0.22855554521083832, 0.7801493406295776, -0.33994099497795105, 2.3577330112457275, 0.7351188659667969, -0.4656635820865631, -1.368698239326477, -1.5968232154846191, 0.9819879531860352, 0.8764678239822388, -0.37704145908355713, 0.5130923986434937, 1.1856944561004639, 1.286374568939209, -0.21445199847221375, 0.1286376714706421, -0.35953348875045776, -1.0083906650543213, -0.6465708017349243, -0.8220996260643005, -0.41590285301208496, -1.5199555158615112, -2.0731804370880127, 1.2380472421646118, 1.7101843357086182, -0.3911799490451813, 0.045795220881700516, 0.07161256670951843, 0.5234678387641907, -0.627051830291748, 1.739073395729065, 1.3533210754394531, 0.09246422350406647, 0.22939719259738922, -1.3907670974731445, 0.2328500598669052, 0.7384675741195679, 0.4357810318470001, -0.6906253695487976, -0.5352108478546143, 0.7903479337692261, -2.159802198410034, 0.6995436549186707, 1.4292511940002441, 0.00334866507910192, 2.954725742340088, -0.33472761511802673, -0.4050877094268799, -0.31187963485717773, 0.48194506764411926, -0.6458655595779419, -0.13266733288764954, 0.5386533737182617, -0.05485443398356438, 0.03492703288793564, -0.13411855697631836, -2.2248828411102295, -0.784959077835083, -0.7087064981460571, -0.4508860111236572, 1.5688563585281372, 0.04548988491296768, 0.7129884362220764, -0.24801212549209595, 0.45270881056785583, 0.24824857711791992, 0.111290343105793, 0.9645194411277771, -0.718444287776947, -1.3681464195251465, -0.33612027764320374, -0.17519502341747284, 1.7533822059631348, 1.3578622341156006, 1.6152650117874146, 1.192240834236145, -0.23057423532009125, -1.6352870464324951, -1.2503820657730103, -0.07476203888654709, 1.3447636365890503, 2.174771308898926, -1.520858645439148, 0.19193905591964722, -0.4521673619747162, -0.7122504711151123, -0.2962264120578766, 0.5015146136283875, -0.30141621828079224, -1.0416935682296753, -0.36450374126434326, 1.0771669149398804, 1.1071292161941528, -0.44000452756881714, -0.654207706451416, -0.05939676985144615, 0.37341779470443726, 1.3163261413574219, -0.9675614237785339, -1.323280692100525, -0.23739494383335114, -0.3839716613292694, 0.32243356108665466, -0.5588902831077576, -1.0033226013183594, -1.410645604133606, 0.2382880300283432, 0.4128812253475189, -0.3392271101474762, -1.4408308267593384, 0.347164511680603, 0.3782458007335663, -0.9282200336456299, 0.21620354056358337, -1.4576075077056885, 0.0750378742814064, -0.2848856747150421, 0.7628337740898132, 0.832487165927887, -1.5558702945709229, 1.1050509214401245, 0.1294766068458557, -1.2304142713546753, 1.141122579574585, 0.9753444194793701, -1.1919739246368408, 1.153649091720581, -0.4388553202152252, -0.18286144733428955, -0.589984655380249, -0.4477296471595764, 0.25830912590026855, 0.35018250346183777, -0.4644513726234436, -3.639026403427124, 0.0392371341586113, -0.6523687243461609, -0.7218068838119507, -0.27461549639701843, -0.16148805618286133, 1.9022969007492065, 1.047705054283142, 2.151367425918579, -0.04992638900876045, 2.503420352935791, 0.5865647792816162, -0.5871029496192932, 0.18805092573165894, 0.27990177273750305, -0.7153692245483398, 1.4412277936935425, -0.058093611150979996, 0.003916917834430933, 1.073678731918335, -1.5373203754425049, -1.0592902898788452, -0.1009717583656311, 0.9570825695991516, 1.5750749111175537, 0.24702218174934387, -1.2574188709259033, 1.7634042501449585, 0.6923431158065796, -0.7753443717956543, -0.11525826901197433, 0.9196728467941284, -0.42536497116088867, 1.0608656406402588, -1.01333487033844, -1.5973780155181885, 0.4568808674812317, 0.11108224838972092, -0.3858266770839691, -1.0403753519058228, -0.23571568727493286, 0.186228945851326, 0.17223691940307617, 1.423947811126709, -0.6197382211685181, 0.4021194577217102, -0.633976936340332, -1.3166515827178955, 0.42716264724731445, 0.15891864895820618, -1.2271510362625122, -0.7421191334724426, -1.5470072031021118, -1.629331111907959, -1.6788612604141235, -0.312024861574173, 0.539677083492279, -0.2887767553329468, -0.10030060261487961, -0.33828023076057434, -0.1648070216178894, -0.8393722176551819, 1.5308424234390259, 0.3602348566055298, -0.39618152379989624, -1.941411018371582, -1.5828434228897095, 0.15570734441280365, -0.2780732810497284, -1.011491298675537, 1.1344972848892212, 0.24894924461841583, 0.2288028597831726, -1.0213629007339478, 0.1615755409002304, -1.023127794265747, 0.37350836396217346, -0.12367039173841476, -0.5685060024261475, -1.0413587093353271, 0.4488750994205475, -0.5285212993621826, -0.696713387966156, -0.27628853917121887, -0.31034430861473083, 0.4483039379119873, 0.8372523784637451, 0.2710326910018921, 0.14088954031467438, -0.8215689659118652, 1.1684815883636475, -0.37228354811668396, -0.25315991044044495, -0.38600605726242065, -0.43630847334861755, 0.31388428807258606, -0.9607892036437988, -1.6113831996917725, 0.005046314559876919, 0.5227446556091309, 0.04080366715788841, 2.061889886856079, 0.832789421081543, -0.09018175303936005, 1.7545028924942017, 1.475008487701416, -0.9012463688850403, -0.7070020437240601, -0.061561163514852524, 0.545478105545044, -0.9297255873680115, -0.06396442651748657, 0.054989442229270935, 0.5625720024108887, 0.23003850877285004, 0.10115637630224228, 1.2646305561065674, -2.6634209156036377, -0.34424370527267456, 1.4015501737594604, 0.010865920223295689, 0.2107701152563095, 1.8513280153274536, 1.1451407670974731, -0.40742939710617065, -1.9783353805541992, -0.4276005029678345, -0.8685094118118286, 0.8873227834701538, 0.1760166734457016, 0.3792932629585266, 1.8106263875961304, -0.7245598435401917, 0.1468842625617981, 0.30098992586135864, 1.5871292352676392, -2.2835190296173096, -1.070706844329834, -1.6038100719451904, -1.3458079099655151, -0.927581250667572, -0.17577122151851654, 1.374054193496704, -0.9491863250732422, 0.17864049971103668, 0.16377802193164825, -0.17047514021396637, 1.288143515586853, -1.306010127067566, 0.1292879581451416, 1.3617606163024902, -0.28236454725265503, 0.39639514684677124, 0.026637157425284386, 0.3842085897922516, 2.145991802215576, -2.1838526725769043, -0.3939536213874817, -0.38455379009246826, -0.09379953891038895, -1.0461468696594238, 1.010594367980957, 0.8290944695472717, 1.2683296203613281, 0.5399820804595947, 1.3747409582138062, -0.2909000813961029, -0.09342565387487411, -0.4886707365512848, 1.4536834955215454, -1.319933295249939, -0.5914580821990967, -2.1637918949127197, 0.6537589430809021, 1.785541296005249, -0.40006914734840393, -0.4198194146156311, 0.0312051959335804, 1.5212140083312988, -1.5365909337997437, 1.3677898645401, -0.9922539591789246, -3.3721652030944824, -0.001952268066816032, -1.609209656715393, -0.7892988324165344, 1.2523263692855835, 1.2597148418426514, 0.07666878402233124, -0.5334191918373108, 0.7213867902755737, -0.16319338977336884, 0.06105997785925865, 1.0445125102996826, -0.45994722843170166, -0.30655157566070557, 0.5914660096168518, -0.16671079397201538, 0.8999197483062744, -1.5125428438186646, 1.0047246217727661, -0.1185973584651947, 0.24322064220905304, -1.5718296766281128, 1.416792392730713, -1.3925367593765259, -0.011169340461492538, -0.4052208662033081, 0.2245999276638031, -0.35752418637275696, -0.84930819272995, -1.2389192581176758, 0.3148709237575531, 0.33129340410232544, -0.694564700126648, 0.4393753111362457, -0.7034531235694885, -2.8533060550689697, 0.11896585673093796, 0.6337444186210632, 0.497199684381485, -1.5568815469741821, -2.1388630867004395, -0.37908661365509033, -0.04490184411406517, 0.5535304546356201, -0.7160589694976807, 0.33074259757995605, 1.1176813840866089, 2.152768135070801, 0.18210673332214355, -1.5000241994857788, 2.71429181098938, 1.1984162330627441, 0.033144354820251465, -1.1492828130722046, 0.017724575474858284, 0.8228192925453186, -0.13915418088436127, -0.2891894578933716, 0.5131678581237793, -0.34130847454071045, -0.6878483891487122, 0.27542030811309814, -0.3237156569957733, 0.06331347674131393, 1.024595022201538, 1.7283499240875244, 0.7188358306884766, -2.0370733737945557, -1.0390326976776123, -1.534111738204956, 0.7870811820030212, -0.17435923218727112, -0.03655999153852463, 0.22886024415493011, -0.21131186187267303, -0.9049708843231201, 0.832331657409668, -1.9132709503173828, 0.8479688167572021, -0.37001973390579224, -1.0051554441452026, 0.9495850205421448, -1.0784189701080322, 1.4646213054656982, 0.40277406573295593, 0.34491845965385437, 0.9174923300743103, -1.4605562686920166, -0.02896502986550331, 0.038866087794303894, 0.9854639172554016, -0.12386110424995422, -1.0073328018188477, -1.5723856687545776, 0.7522255182266235, 0.10460970550775528, -1.465698003768921, 1.275027871131897, 0.8317279815673828, -0.2571716904640198, -2.1108880043029785, 0.7388744950294495, 0.8530616164207458, 0.5711484551429749, 0.8301267027854919, 0.9735803008079529, -0.6717640161514282, -0.8935317397117615, 1.7569206953048706, 0.06719768792390823, -2.0675244331359863, -0.417368084192276, 1.329958200454712, -1.9359675645828247, 0.518386721611023, -0.7300049662590027, 0.37527939677238464, -2.6695547103881836, 1.1021791696548462, -1.0742981433868408, -0.018917622044682503, 0.37159574031829834, -1.0799918174743652, 0.11649426072835922, -1.8189799785614014, -1.0317264795303345, 1.8110660314559937, -1.3484421968460083, -1.958628535270691, 0.4133112132549286, -0.09170819818973541, 1.3655781745910645, -1.0002435445785522, -0.9710686802864075, 1.2780444622039795, -1.8236165046691895, -0.7873744964599609, -0.41535356640815735, -2.3456034660339355, 0.18577289581298828, -0.040518809109926224, -0.10166782140731812, 1.5741195678710938, 0.008245471864938736, 0.20501355826854706, 0.09151641279459, 0.3051098883152008, -0.2662855386734009, 1.1324458122253418, 0.5867773294448853, 1.73833167552948, -0.2918379008769989, 1.3212767839431763, 2.1603426933288574, 1.792898416519165, 0.9950897693634033, -1.7193132638931274, -0.501762330532074, 0.6927226781845093, 1.1169824600219727, 0.40706393122673035, -1.6272755861282349, -0.7182453870773315, -0.4608357846736908, -0.49454453587532043, 0.7661111950874329, 0.9001768231391907, -0.4020969867706299, -0.3522160053253174, 0.5711458921432495, 0.7909429669380188, -1.1290894746780396, 0.29253488779067993, -0.4159431457519531, 1.1213324069976807, -0.18723393976688385, 1.4174977540969849, 0.45771750807762146, 0.6510952711105347, 0.8364070057868958, 1.0325953960418701, -0.8613244891166687, -2.006574869155884, -0.7756134867668152, 0.31689244508743286, -0.8283915519714355, -0.9944217801094055, -0.4030679762363434, 1.5091726779937744, 0.3449867367744446, 1.261300802230835, -1.0922108888626099, 0.25808271765708923, -0.49563664197921753, -0.1551787257194519, -0.9210290908813477, -1.0546289682388306, 1.0158921480178833, -0.6004903316497803, 0.3833921551704407, -0.5630015730857849, 0.38832035660743713, -0.7506044507026672, 0.18509706854820251, -1.5436604022979736, 0.9595302939414978, -1.0396854877471924, -0.9566935300827026, 0.18167895078659058, -0.01061955839395523, 1.0449552536010742, 2.5253489017486572, -0.3090658187866211, -1.4968141317367554, -1.106918215751648, 0.8912749886512756, -0.45526888966560364, -0.513289213180542, -2.5788347721099854, 1.9406638145446777, 0.3025727868080139, -0.3037736713886261, 1.2540205717086792, -1.5366291999816895, -1.3066085577011108, -0.7341743111610413, 1.6058658361434937, -1.0805425643920898, 2.6875436305999756, 1.6651908159255981, 0.017364555969834328, -2.1510043144226074, -1.8301819562911987, -1.4017914533615112, -1.458249568939209, 0.17188464105129242, 0.280669242143631, -1.1773102283477783, 0.43837347626686096, -1.2731921672821045, -1.2522492408752441, -1.4966189861297607, -0.250449538230896, 1.2014812231063843, 0.07011155784130096, -0.11536750942468643, -0.010117176920175552, 0.7873867154121399, -0.5264137983322144, 1.1834468841552734, -0.10449331253767014, -0.321216344833374, 0.3791598379611969, -0.5740926861763, 1.3521682024002075, 0.263746976852417, 0.5108803510665894, -1.8855918645858765, 1.7784545421600342, -1.5757352113723755, -0.917905330657959, -0.48064327239990234, 1.9281260967254639, -0.332263708114624, 0.9225354790687561, -0.30820226669311523, -0.02721032314002514, -0.19339150190353394, -0.47321397066116333, -1.0495458841323853, -0.204549640417099, 0.9334242343902588, -0.9941304922103882, -1.1467087268829346, -1.113872766494751, -0.25023213028907776, 1.713494896888733, 0.016057653352618217, -0.21035413444042206, -0.1407753825187683, -1.7473671436309814, 0.41216734051704407, 1.515902042388916, 0.18656834959983826, -1.106856346130371, -0.13370704650878906, -0.4909941852092743, -0.33048215508461, 0.02136741206049919, 1.1653895378112793, -1.072888970375061, 1.4194188117980957, -1.4378294944763184, -1.047189712524414, -0.5483022928237915, 0.4829898774623871, -1.0502710342407227, 0.5164602994918823, 0.9971203804016113, -0.4876938760280609, -0.24628442525863647, 0.6240825653076172, 0.24053196609020233, -0.26764991879463196, -2.3088061809539795, 0.6954973340034485, 1.2856594324111938, -0.5971055626869202, 0.38120532035827637, -2.174586057662964, -1.1197023391723633, -0.03855188563466072, 0.6901636123657227, -1.4898208379745483, -0.4257112741470337, 0.4780552387237549, 1.2416857481002808, -0.2876666784286499, -0.6393312811851501, 0.18265677988529205, 0.8778060674667358, -1.4729341268539429, 0.8305824398994446, -1.649051308631897, -0.6316565275192261, 1.4723608493804932, -1.9768413305282593, -0.0431867390871048, 1.2755579948425293, 0.8996332883834839, 1.1492102146148682, -0.8032751083374023, 0.22526022791862488, 0.4333178699016571, 0.42358431220054626, -0.34373512864112854, -0.028906358405947685, 0.5859508514404297, -0.39627546072006226, -0.2500374913215637, -0.19473719596862793, -0.2153095304965973, 1.8647010326385498, 0.7498400211334229, 0.5914983153343201, 0.96245938539505, -0.4473447799682617, -0.5754852890968323, -0.2121095508337021, -0.5055828094482422, -0.07381436228752136, -1.4580368995666504, 2.1049845218658447, -2.7511065006256104, 2.576842784881592, -0.6282944083213806, -1.8963605165481567, -1.4958040714263916, -0.03778450936079025, 1.1576228141784668, 1.7539371252059937, -0.36504361033439636, -0.22195520997047424, -0.21006622910499573, -0.6877322793006897, -1.8231666088104248, 0.8368117809295654, -0.826151967048645, -0.894441545009613, -1.3821687698364258, -0.28039562702178955, -0.29990383982658386, -1.194942593574524, 0.9794081449508667, -0.4081569015979767, -0.6560592651367188, 0.002181188203394413, 1.299620270729065, -1.7639520168304443, 0.9662957191467285, -0.16346800327301025, -1.1789932250976562, 0.8246965408325195, 0.8503364324569702, -0.7260683178901672, 0.40694889426231384, -1.091546893119812, 0.7136209011077881, 0.7739627957344055, 0.28899088501930237, -0.49605584144592285, -1.678590178489685, 2.2297234535217285, 0.9056186079978943, 0.9631413817405701, 1.220139741897583, 0.08458390086889267, -1.683917760848999, 0.885883092880249, -1.334798812866211, 1.3461530208587646, 0.5858514904975891, 0.008885277435183525, -0.11887781322002411, 1.5004962682724, 0.7422794103622437, -0.5173248052597046, -0.42182278633117676, -0.7195801734924316, 2.9623987674713135, -0.528777539730072, -0.42655929923057556, 0.8704551458358765, -1.393053650856018, -0.0854467824101448, -0.48671019077301025, 0.8133094906806946, -0.3618418276309967, 0.46444571018218994, -0.42763787508010864, 0.11141666769981384, 0.20055583119392395, 0.6426605582237244, -0.06158418580889702, -0.24773383140563965, -1.7183845043182373, 1.4560109376907349, 0.5750851631164551, 2.2156834602355957, -0.29457440972328186, -0.4174334704875946, -0.2833927273750305, 0.089333176612854, 0.3673135042190552, -1.1933763027191162, 0.3114885091781616, -1.0838932991027832, -0.1856093406677246, -1.378296136856079, 0.09814354032278061, -1.4063013792037964, -0.5321717858314514, 0.3095577359199524, -0.0495019368827343, -1.4906396865844727, 1.219143033027649, 1.0587822198867798, -0.14066217839717865, 2.0403923988342285, -1.0429672002792358, 0.19952911138534546, -1.0766677856445312, 0.8923420310020447, 0.9421340227127075, 1.0329742431640625, -0.9485159516334534, 0.2987493574619293, -0.9867107272148132, 1.4712717533111572, -0.5348862409591675, 0.0031355328392237425, -0.47972559928894043, -0.266514390707016, -0.9507763981819153, -1.7765179872512817, -0.5750088691711426, 0.14287498593330383, -0.6443917155265808, 0.046863991767168045, 0.35423266887664795, -0.11670905351638794, -0.4685598909854889, -1.1555476188659668, 1.9451137781143188, -0.7326100468635559, -0.3468030095100403, -1.23593008518219, 0.12813617289066315, 0.35214975476264954, 0.7088223695755005, 2.4396800994873047, -1.4902443885803223, 1.169568419456482, 1.145678997039795, -0.8356676697731018, 0.1385394185781479, 2.565492868423462, -0.5945490002632141, -0.9638134837150574, 1.2658824920654297, 0.32945170998573303, -0.3151973485946655, -0.39612820744514465, -1.587035059928894, 0.030049853026866913, 1.1709461212158203, -0.990566611289978, 0.9896735548973083, 1.4001928567886353, -0.48289814591407776, 0.1958969086408615, 0.6670194864273071, 0.5996365547180176, 0.824203610420227, -2.0289313793182373, 0.257485955953598, -0.19254596531391144, -0.45025181770324707, 0.6501538157463074, 1.2987288236618042, 0.7953696250915527, 0.23900890350341797, 0.7766619324684143, 0.9506909847259521, 0.7790993452072144, 0.4918579161167145, 0.5097584128379822, 1.1833653450012207, -0.5447829961776733, 2.134389877319336, -0.1877998560667038, 2.1117215156555176, -0.40424644947052, 1.1068817377090454, -0.09710181504487991, 0.6508513689041138, 0.27312231063842773, -1.1176048517227173, -0.3060299754142761, -1.0211297273635864, -0.22894984483718872, -0.6935104727745056, -0.2123953402042389, -0.30956006050109863, 1.6785041093826294, -1.0048283338546753, -0.3996295630931854, 0.6691423058509827, 1.3470923900604248, 0.46751171350479126, -0.3334283232688904, 0.9117929339408875, 0.9096297025680542, 2.031522035598755, 1.3440741300582886, 0.4672073423862457, 0.4261385202407837, 1.8626378774642944, 1.3515558242797852, -1.581012487411499, -1.3083504438400269, -0.8128530979156494, 2.5584309101104736, -1.388443112373352, 0.4046918451786041, 0.3158809542655945, -0.7599875926971436, -0.9436249732971191, 0.8467758893966675, 0.2824265658855438, -1.1091543436050415, -1.129128098487854, 0.07548517733812332, -1.296506643295288, 0.9234639406204224, 0.6714411377906799, -1.3728910684585571, 2.1879615783691406, -0.8711367249488831, -0.4873095452785492, 0.27775949239730835, -0.5757021903991699, 0.06690533459186554, -1.2544488906860352, 1.065536379814148, -0.4144270718097687, -0.032491397112607956, -0.9687965512275696, 2.3966891765594482, 0.09220698475837708, -0.26372724771499634, 0.3115609288215637, 0.956645667552948, -2.0550830364227295, 0.01765606366097927, 0.06343559175729752, -1.1870497465133667, -2.2874815464019775, -1.0884042978286743, 0.19849136471748352, 1.4969816207885742, -0.2611199617385864, -0.11938344687223434, 0.6525663137435913, 2.033708333969116, -1.5257705450057983, -1.2280757427215576, -0.3629690408706665, 1.5471121072769165, -0.649976909160614, -1.8020540475845337, 1.7717679738998413, -1.2491395473480225, -1.9806931018829346, 1.2189793586730957, 0.3347938358783722, 0.05151825398206711, 0.41879522800445557, 3.066387414932251, -0.6543105840682983, 0.14359617233276367, -0.16576968133449554, 1.476600170135498, -1.068569540977478, 1.6708840131759644, -0.049265582114458084, -0.3975497782230377, 0.9623923897743225, -0.588095486164093, 0.5095856785774231, 0.14855705201625824, 0.07293229550123215, 0.6931708455085754, 0.026531005278229713, 1.1762189865112305, -1.100995659828186, 0.1315264254808426, -1.5878334045410156, 0.7105267643928528, -0.09951599687337875, 1.5927646160125732, -0.19811901450157166, -0.27970343828201294, -0.22440294921398163, 0.4311245381832123, 0.5925745368003845, -0.5600107312202454, -1.9364203214645386, -0.029459886252880096, 0.5987527966499329, -1.6320509910583496, -0.5503419637680054, -1.3361252546310425, 1.587241530418396, -0.6548294425010681, 1.2962517738342285, 0.0844796672463417, 1.5554968118667603, 0.03259226679801941, -0.3722343444824219, -1.8753820657730103], [-0.3752604126930237, 0.09203150123357773, -0.6879382133483887, -0.08568155020475388, -1.4568029642105103, -0.21528097987174988, 2.095402956008911, 0.634803056716919, -0.8888907432556152, 0.2659037709236145, 1.5889124870300293, 0.07434310764074326, 0.4949186146259308, 0.16293281316757202, 1.145388126373291, 1.6799594163894653, 0.34850314259529114, 0.5324859023094177, 0.34378162026405334, 0.0941503643989563, -0.6296889185905457, -0.12095138430595398, -2.0457849502563477, 1.6978212594985962, 1.0498768091201782, -2.576801061630249, -0.6143867373466492, -0.5755424499511719, 0.37780940532684326, -1.405493974685669, 0.9163756370544434, -0.19654330611228943, -0.06476062536239624, 1.8480467796325684, -0.05326751992106438, -0.12062322348356247, 0.32328397035598755, 0.5567051768302917, 2.1189918518066406, 0.4435156583786011, -1.682268738746643, -0.026615256443619728, 1.5585514307022095, -1.1469449996948242, -0.004056757315993309, -2.390533447265625, 0.5573792457580566, 0.7909559607505798, -0.545366644859314, -1.3506317138671875, 0.7703841328620911, -0.09186668694019318, -0.7354408502578735, 0.7040693163871765, -0.650826632976532, 0.24301105737686157, 0.8208222985267639, -1.1852790117263794, 0.27607643604278564, -0.45116737484931946, 0.1824764460325241, -1.2016946077346802, -0.11959312856197357, 0.7498489022254944, 0.051414262503385544, 0.2547401487827301, -1.3093416690826416, -0.11690369993448257, -0.2655703127384186, 0.6708347201347351, -0.7920419573783875, 0.2626807391643524, 0.9474327564239502, -0.641637921333313, 0.2350703924894333, 0.44589462876319885, 1.1133332252502441, 0.251652330160141, 0.40456652641296387, 0.13340795040130615, -1.9900518655776978, -0.9065046906471252, -0.035723548382520676, 0.550503134727478, 1.4525835514068604, 1.2191438674926758, 1.1334210634231567, -0.23613081872463226, 0.5136423707008362, -0.44521477818489075, -0.9026762843132019, 0.6391419768333435, -1.2249609231948853, 1.94869065284729, -1.3110440969467163, 0.20333583652973175, 0.5220482349395752, -0.30449479818344116, 0.29085198044776917, -0.0737307220697403, 0.08702176809310913, -1.0300793647766113, -0.19263868033885956, 0.40156567096710205, 0.5124688744544983, -0.22714626789093018, 0.2764808237552643, -0.9444381594657898, 1.2376465797424316, 1.145994782447815, -1.3089687824249268, 0.6430174112319946, -0.11142352968454361, -0.16382357478141785, 0.5291982293128967, -0.11744874715805054, -1.2169348001480103, -0.6739420890808105, -0.3804713487625122, -0.306966632604599, -1.374103307723999, -0.1831166297197342, 0.6262378692626953, 0.7332707643508911, 0.14550791680812836, -0.1729728728532791, 0.39195096492767334, 0.009795536287128925, 2.0225865840911865, 0.990827202796936, 0.16333162784576416, -0.10997379571199417, -0.6815710663795471, 2.0698747634887695, 1.9170371294021606, 1.7387454509735107, 0.3101847171783447, 0.293865829706192, -0.2912430167198181, -0.7135443687438965, -0.08029288798570633, -0.47758767008781433, -0.897894024848938, 0.6164449453353882, -1.3961998224258423, 0.13810449838638306, -0.8768770098686218, -0.4222848117351532, 0.4102659225463867, -0.5363627076148987, -0.2557052671909332, 0.27306678891181946, -0.22167056798934937, 1.2449253797531128, 0.9371933341026306, -0.31017062067985535, -0.7987905740737915, 0.4469459056854248, 2.107640266418457, -0.6602592468261719, -0.7510225772857666, -0.9680138230323792, -1.4414047002792358, -1.7687193155288696, 0.6817575097084045, -1.1746985912322998, -0.48322391510009766, -0.4481615126132965, 1.5798698663711548, -0.9573583602905273, -0.49816203117370605, 0.7695786952972412, -1.9649417400360107, -0.4307846426963806, -0.9193816184997559, 0.4084278345108032, 2.0065560340881348, 1.2620288133621216, -0.20688925683498383, -0.9551790356636047, -1.2455540895462036, -0.21488820016384125, 0.7140395641326904, -0.006257177330553532, 0.04011330381035805, 1.217828631401062, 0.8317291140556335, 1.3352704048156738, 0.7742725014686584, -0.056790225207805634, -0.2943563461303711, -0.09662659466266632, -0.9243636131286621, 1.1876686811447144, -1.1652703285217285, -0.32804760336875916, -1.7697020769119263, -0.5213770270347595, -0.5255752205848694, -0.32027196884155273, 0.6247261762619019, -1.0450676679611206, -0.38681545853614807, 0.9773853421211243, 0.40488505363464355, -0.3506968021392822, -1.285696029663086, -0.4198126792907715, -0.931879997253418, -2.3804585933685303, -0.27576667070388794, -1.5130590200424194, 0.8861295580863953, 1.2517882585525513, -1.3123916387557983, 0.4264151155948639, 0.7529798150062561, 0.9434033036231995, 0.8587801456451416, 0.5421591997146606, -0.3616701066493988, -0.8414687514305115, -0.7587650418281555, -1.7188925743103027, 0.4468500018119812, 0.3219417929649353, -2.4084131717681885, -0.3979569971561432, -0.10313291847705841, 0.568953812122345, -2.795069456100464, -0.3402538597583771, -0.913060188293457, -0.31870242953300476, -1.25331711769104, 2.1564972400665283, -1.3303028345108032, -1.7633001804351807, 0.39306557178497314, -0.1531250774860382, -2.379643678665161, -0.11427205055952072, -1.4403327703475952, 0.8994626402854919, 1.0089908838272095, 0.7694200873374939, 1.3734160661697388, -1.2939555644989014, -0.18397186696529388, -0.7489811778068542, -0.5333564281463623, -0.17196060717105865, 0.28774553537368774, -1.0361744165420532, 0.35156866908073425, -0.4649360775947571, -0.8527441024780273, -1.0732910633087158, -0.9390227198600769, -0.33391642570495605, 1.1288830041885376, -0.16522423923015594, -1.390403151512146, -0.4655471444129944, 0.34943801164627075, -0.11884670704603195, -0.015379718504846096, 0.31582605838775635, 0.21986985206604004, 2.3503611087799072, 0.1462605595588684, 2.5357885360717773, -0.3097919821739197, -0.4426683783531189, -0.4518696963787079, 0.19670705497264862, 1.8370535373687744, 0.001638793502934277, -0.17962947487831116, -1.2516913414001465, -0.5394644141197205, 0.6227290630340576, 0.5326897501945496, 2.619950771331787, 1.3828747272491455, -0.27179691195487976, 1.4667352437973022, -0.8405770659446716, -0.19626256823539734, -0.15837596356868744, 2.192092180252075, 1.3833550214767456, -0.7634367346763611, 0.17231272161006927, 0.7702524662017822, 0.8198302984237671, -0.7895368337631226, 1.4288126230239868, -1.9831167459487915, 1.2231308221817017, -0.4857918918132782, 0.8327940702438354, 2.634873628616333, 0.4983666241168976, -0.7022963762283325, -0.2519110143184662, 1.0264487266540527, 0.7660232186317444, -0.7868849635124207, 0.9927680492401123, 0.4557316303253174, -2.8264236450195312, -2.0695786476135254, -0.44119036197662354, 0.21120263636112213, -2.0434064865112305, 0.696006178855896, 1.462162971496582, -1.6496943235397339, 0.3761756122112274, -0.3122532367706299, 0.4499724209308624, -0.33216047286987305, 0.237177312374115, -0.13026636838912964, -0.688784122467041, 0.2818242013454437, -0.013061520643532276, 1.9912649393081665, 0.6901518106460571, 1.2761722803115845, -0.9717357158660889, 1.5488061904907227, -0.02225540578365326, -0.6906057000160217, -2.1169815063476562, -0.3619062900543213, 0.7374230623245239, 2.215769052505493, 0.6729127764701843, 0.9101110100746155, -0.6164249777793884, -1.204211711883545, -0.8074372410774231, 1.0213673114776611, 0.7298291325569153, 1.492120623588562, -1.128866195678711, -0.2106814682483673, -1.6276973485946655, 0.7316235303878784, -0.8533012270927429, 0.1975153088569641, -1.0289593935012817, -0.7837840914726257, -1.0652074813842773, -0.2613561153411865, 0.056193362921476364, -0.6606593132019043, 2.154914140701294, 0.6729092597961426, -1.2838178873062134, -0.527547299861908, -1.3190860748291016, -1.2042065858840942, -0.5374425649642944, 0.7793334126472473, -0.22617323696613312, 0.5971488356590271, 1.1762968301773071, 0.1702902913093567, 1.5675677061080933, -1.119235634803772, -0.5950467586517334, 0.49211612343788147, -0.8034385442733765, 0.1600465029478073, -0.5882281064987183, 0.3961082100868225, 2.627671480178833, -0.3684844374656677, 1.3542284965515137, 0.1269054263830185, 0.40452808141708374, 2.1256113052368164, -1.220223307609558, -0.39667433500289917, 0.39702796936035156, 0.1595596820116043, -1.1762707233428955, 0.0786346048116684, -1.286068320274353, -0.820925772190094, -1.2671353816986084, -0.45475542545318604, -1.2806975841522217, -0.03247528523206711, -2.168186664581299, 0.7866137027740479, -0.38774576783180237, -0.5263469219207764, 1.2820782661437988, 1.464555263519287, 0.079831063747406, -0.0707089900970459, -0.10740497708320618, -0.20127835869789124, -2.044816255569458, 0.1886889934539795, -0.9700962901115417, 0.8495252132415771, -1.338212490081787, -0.03126010671257973, -0.42661306262016296, 0.27175456285476685, 1.5967942476272583, -0.4817678928375244, -2.2454283237457275, -1.177955985069275, -0.20206551253795624, -0.14132215082645416, -0.06605812162160873, -0.43549853563308716, 1.4941326379776, -0.14621463418006897, 0.5995352268218994, -0.440615713596344, 0.8669129014015198, -0.17711541056632996, -0.28296926617622375, -0.7328618168830872, -1.1117897033691406, 0.9793959259986877, 0.3350190818309784, -1.594599723815918, 1.2000911235809326, 1.227298617362976, 1.1288652420043945, -0.2563680410385132, 0.8301827907562256, -0.390546590089798, 1.7168322801589966, -0.8103591799736023, -0.1728607565164566, 1.8152533769607544, -1.1830610036849976, -0.5959270596504211, -0.9299365878105164, -0.8734483122825623, 1.4169949293136597, -1.060226321220398, 1.7303892374038696, 0.46289125084877014, -2.065635919570923, -2.2879445552825928, 0.6784915924072266, -0.3779062330722809, -1.2116756439208984, -1.753533124923706, 1.1547836065292358, -0.3403071165084839, 0.4932705760002136, 0.24103428423404694, -1.2963062524795532, -0.44143548607826233, -0.2908742129802704, 0.6043033003807068, 0.46280208230018616, -0.029493024572730064, 0.0828973725438118, 1.0460330247879028, -0.19182436168193817, 0.3477070927619934, -1.2500314712524414, 0.24298077821731567, 1.1525964736938477, 1.234468698501587, 0.20362946391105652, -0.3875081241130829, 1.296204924583435, -0.1303337961435318, -0.007970922626554966, 0.7833726406097412, 0.30294638872146606, 0.38033801317214966, 0.8290307521820068, 0.55326908826828, -0.4658079743385315, 0.43573078513145447, 0.8938713073730469, -0.8224053978919983, -0.054090745747089386, 0.026209352537989616, 1.2487080097198486, 0.5134055614471436, -0.3987453579902649, -1.0383062362670898, -1.0905015468597412, -0.7002814412117004, -0.3556390702724457, -0.074652299284935, 1.588680624961853, 0.18811966478824615, 1.3759487867355347, -1.1829729080200195, 0.17666485905647278, 0.11790428310632706, 0.46481937170028687, -0.5092380046844482, 2.2463040351867676, -1.0922658443450928, 0.8676872849464417, 0.7800992727279663, -2.0788209438323975, -1.0578757524490356, 0.4893723130226135, 0.3432941734790802, -0.08479625731706619, 0.21409566700458527, -0.2904815077781677, 1.5944679975509644, -1.6181241273880005, -0.4451185464859009, 0.46715033054351807, 0.26171645522117615, 0.7676350474357605, -1.8417476415634155, 1.5504909753799438, -0.547085165977478, -0.26116082072257996, -0.9923635721206665, 0.33548906445503235, 0.11503181606531143, -0.665455162525177, 0.8738057613372803, 1.7077386379241943, -1.5297058820724487, -0.9811761379241943, 1.2032959461212158, 1.173550009727478, -0.9529512524604797, 0.09202983975410461, -0.30168744921684265, 1.4417006969451904, 0.005292421206831932, 0.5361765027046204, 0.6390668749809265, -0.06837988644838333, -0.5626255869865417, 0.7607279419898987, -0.06691955775022507, 0.24422132968902588, -0.2220764309167862, -0.00428591575473547, 1.5594027042388916, -0.04872831329703331, -0.14274193346500397, -1.5789355039596558, -2.398826837539673, 0.19691048562526703, -0.41030818223953247, 0.1333482563495636, -0.08773422986268997, 1.1151052713394165, 1.6795289516448975, -1.6213619709014893, -1.03620183467865, 1.7143678665161133, 0.09390751272439957, -0.35382193326950073, 0.7053877711296082, -0.6690627336502075, -0.44203492999076843, 0.7895026803016663, 1.9702852964401245, -1.9263311624526978, -2.1366589069366455, -0.6641154885292053, -0.2424919158220291, -0.5552859306335449, 0.5709114074707031, -0.030783016234636307, 0.3195572793483734, 1.2256908416748047, 1.247955322265625, 1.1435781717300415, -1.8794225454330444, -0.35236409306526184, 0.7833629846572876, -1.2598814964294434, 0.5035169124603271, -0.33148393034935, 0.0830189660191536, -0.47208306193351746, -0.26124265789985657, 0.5832886099815369, 1.3958659172058105, 0.07879997789859772, -1.0277913808822632, 1.0912226438522339, -0.904050886631012, 0.9076164364814758, 0.29430916905403137, -0.8000466227531433, -1.2286648750305176, -0.3912332057952881, 1.0764479637145996, -0.16250108182430267, -0.13676749169826508, 2.2152132987976074, 1.1239477396011353, -1.0341228246688843, 0.5375053286552429, 0.6468539237976074, 1.470503330230713, -0.5137219429016113, 0.27515777945518494, -0.14328326284885406, -0.1589565873146057, 0.4830363690853119, -0.616347074508667, -1.1211570501327515, 1.5214580297470093, -2.4583678245544434, 0.3047686815261841, 1.2993056774139404, -0.15271449089050293, -1.9637185335159302, 1.023986577987671, 0.08859522640705109, -0.11367293447256088, -0.6596775650978088, -1.1951950788497925, 0.5964274406433105, 1.0026601552963257, 0.18539932370185852, -1.6234813928604126, -1.7686597108840942, 0.43553948402404785, 0.4838634133338928, 0.6756389141082764, -0.01836540549993515, 0.96583491563797, 1.1519107818603516, 0.7509692907333374, -0.8928167819976807, -0.7775589227676392, 1.0361055135726929, 0.7396585941314697, 1.1428632736206055, -1.0286991596221924, -0.5879024863243103, 0.9470744729042053, -0.7971503138542175, 0.6340758204460144, 1.7956295013427734, -0.5063889622688293, -0.7496507167816162, -0.8327760696411133, -0.14446710050106049, 0.0020374851301312447, -1.454933524131775, -0.15334221720695496, 1.1561965942382812, -0.5711938738822937, 1.7219923734664917, 0.36295783519744873, -2.1681904792785645, -1.1793205738067627, 0.7811960577964783, -1.9092060327529907, -0.4206726849079132, -0.37948647141456604, 1.4139084815979004, -0.9081326127052307, 0.6116340756416321, -0.3639945685863495, 1.3459762334823608, 0.8731724619865417, 2.13392972946167, 0.5956900715827942, -0.06373871862888336, -0.2473832368850708, 0.08208822458982468, -0.5614305734634399, -1.4015634059906006, 2.4837965965270996, -0.601881206035614, -0.04728352278470993, 0.7270697951316833, 1.2783422470092773, 0.48258593678474426, -0.8104040026664734, 0.3856337070465088, -1.1322098970413208, -0.40416279435157776, 0.592004120349884, -1.2024738788604736, -0.2665809988975525, -1.4837809801101685, -0.021561842411756516, -0.327962189912796, 0.45119866728782654, 0.9979181289672852, 0.3670846223831177, 0.861640989780426, -1.0397928953170776, 0.28004080057144165, 1.5374395847320557, -0.9107518196105957, 1.329329013824463, 1.7252453565597534, -0.4510156810283661, 0.1624942570924759, -0.3237903118133545, 0.4404507875442505, 0.7572662234306335, 1.760452151298523, 2.630993127822876, -2.9804763793945312, -0.3118923306465149, -1.0615648031234741, -0.3799206018447876, -1.2717041969299316, 0.2502870261669159, 0.5478770732879639, -0.953191876411438, 0.28435376286506653, -1.1984810829162598, -0.8729948997497559, -0.4773833155632019, -1.290953278541565, -0.9525985717773438, -0.7938001155853271, 1.2771947383880615, -0.46028077602386475, 0.029575910419225693, -0.00863563735038042, 1.2759382724761963, 1.592566728591919, 1.3529504537582397, 1.6360901594161987, 0.536544919013977, 0.23669105768203735, 0.4206210672855377, 0.6295803785324097, -0.5866578817367554, -1.681727409362793, -0.3262326419353485, -0.6840493679046631, 1.1002418994903564, 1.6005836725234985, -0.593989372253418, -1.46608304977417, 0.14520025253295898, -0.49767574667930603, -0.64356929063797, 0.4996039867401123, 0.3916577696800232, 1.3062752485275269, 2.8563528060913086, -0.18574431538581848, 1.6000478267669678, -0.3514029085636139, -0.37042665481567383, 0.007353866472840309, -1.5026732683181763, -1.6858817338943481, -0.5762351155281067, -0.37631455063819885, -2.2282731533050537, 2.766279697418213, 0.9590499401092529, 0.8878769278526306, 1.3350437879562378, -0.7027486562728882, 0.5677998065948486, 2.280073881149292, 0.44820666313171387, -1.4493350982666016, 1.2439075708389282, 0.6730657815933228, -0.678105354309082, -0.7453784942626953, 0.11921276152133942, 1.677483320236206, 1.036082148551941, -1.4605658054351807, 0.17068593204021454, 0.25822049379348755, 1.2395881414413452, 0.608640193939209, 0.07021678984165192, 1.6267670392990112, 1.3015329837799072, -0.8271288871765137, 1.0070818662643433, 0.9450664520263672, -1.3577227592468262, 0.25415289402008057, -0.06170390918850899, -0.0012122576590627432, 0.5793673396110535, 1.5619159936904907, 0.905085027217865, -0.1888178437948227, 0.020952235907316208, -0.4201076626777649, -0.9277089238166809, 0.13794688880443573, -0.44559407234191895, 0.9365483522415161, -1.359139084815979, 0.6094841361045837, 0.03645945340394974, 0.9120819568634033, 0.08147867769002914, 0.04094463959336281, 0.2427680790424347, 0.10555700957775116, -0.2547031044960022, -2.1581571102142334, 0.43135976791381836, -1.8138877153396606, 0.29995179176330566, 0.1663876473903656, -1.1404162645339966, 0.9411757588386536, -1.9531445503234863, -0.8519208431243896, -0.7316825985908508, 1.3189728260040283, 0.41072213649749756, -1.037611484527588, 0.0866551473736763, -0.6993451118469238, 0.5197339057922363, 2.0692522525787354, -0.8426808714866638, -1.3664911985397339, 0.6165310144424438, -0.2649303078651428, 0.7535758018493652, 0.7623972296714783, -1.7293719053268433, -0.06627073884010315, -0.3942824602127075, -1.5239949226379395, 0.7653292417526245, -0.7038800120353699, 1.4190114736557007, -0.24132020771503448, 1.5090258121490479, 0.6081616878509521, 0.859228789806366, 0.1314934492111206, 0.313062459230423, 0.5678449869155884, -0.765471875667572, -0.5732356905937195, 0.06224999204277992, 0.13134494423866272, 0.10150426626205444, -0.90420001745224, 0.4777262508869171, 0.5379967093467712, -1.4431350231170654, 0.1399431675672531, 0.1441066414117813, -0.22121962904930115, -0.9063704013824463, 1.0787192583084106, 0.1132536455988884, 0.04757216200232506, -0.433258980512619, 0.4390912652015686, -0.03640800714492798, 0.9011809229850769, -1.8109444379806519, -1.4207075834274292, -1.8892779350280762, -0.05005054175853729, 0.44283151626586914, 0.4018896520137787, -0.6787681579589844, -0.19830018281936646, 1.9353090524673462, 0.5085316896438599, -0.09744712710380554, 0.31645190715789795, 0.5299771428108215, 0.17243137955665588, 1.4401252269744873, 0.14626197516918182, -0.006125304382294416, -1.2641584873199463, -1.3274093866348267, -0.7259722948074341, 0.6624645590782166, 1.430987000465393, -0.11950333416461945, -0.49418285489082336, -0.6493059396743774, 3.048342704772949, -0.19421102106571198, 0.7176578044891357, 0.2259339988231659, 0.5601987838745117, -1.4200854301452637, 0.3554248511791229, -0.3162647783756256, -0.703683614730835, -0.22630895674228668, -1.308417558670044, -0.7217877507209778, 0.7363134622573853, 0.06792748719453812, -0.05447939410805702, -1.4568437337875366, -2.1466712951660156, -1.1166760921478271, -0.7527117133140564, -0.6764271259307861, -0.08449285477399826, 0.08656078577041626, -0.12538181245326996, -1.301365613937378, -0.48725754022598267, -0.5301375389099121, 0.5098475813865662, -0.1956377923488617, -1.0930888652801514, -1.6726791858673096, -0.3424178957939148, 0.6992850303649902, -0.14617475867271423, 1.4583408832550049, -0.973941445350647, 0.5278345942497253, 2.682168960571289, -1.0774660110473633, 0.9802725911140442, -0.23884502053260803, 0.9348809123039246, -0.12716332077980042, -0.19752748310565948, 0.7821992039680481, 0.9671345949172974, -1.6118069887161255, 1.9776897430419922, -0.8939559459686279, -0.5836256146430969, -1.0689142942428589, -1.3495744466781616, -1.0167773962020874, 0.05412028357386589, -0.25660985708236694, 0.3604622185230255, 0.848440945148468, 0.11087790131568909, -0.3923420310020447, 1.4829761981964111, 1.2518783807754517, -1.7116937637329102, 0.17599348723888397, -0.1753663867712021, 0.4306517243385315, 0.030623728409409523, 1.217481255531311, -0.15130455791950226, 1.597487449645996, -0.6178978085517883, -0.4434099793434143, 0.5496411323547363, 1.79433274269104, -1.5057142972946167, 0.9721684455871582, 1.4509069919586182, -0.5472508668899536, 0.0457778237760067, 0.224530428647995, 0.8602771759033203, -0.32105469703674316, 0.6390687823295593, -0.2062196284532547, 0.7146257162094116, -0.6461060643196106, -0.09546013921499252, 0.07861325144767761, -1.2098636627197266, 0.10609184205532074, -0.3677758574485779, -0.5411838889122009, 1.5396158695220947, -0.3771266043186188, 0.1099088042974472, -0.3312529921531677, -0.05897098034620285, -0.00240863929502666, 0.4117153584957123, -0.006403041072189808, 1.4277076721191406, -1.199383020401001, 0.07379530370235443, -0.030739676207304, 0.11317935585975647, 0.9507865905761719, 1.6818873882293701, -0.9600902199745178, 0.1434304267168045, -1.4308421611785889, 0.28765490651130676, -0.21377955377101898, -1.4223532676696777, -0.3037105202674866, -0.1447061002254486, 0.2122804969549179, 0.7226470112800598, 0.37049245834350586, -0.16149576008319855, -2.210343599319458, 0.04084470123052597, 0.6673300862312317, -0.8463765382766724], [-1.0210692882537842, 0.5946669578552246, 0.5871862173080444, 0.5217451453208923, 0.21411310136318207, -0.7685157060623169, -0.22971822321414948, -0.14668433368206024, -0.47272855043411255, 2.069039821624756, 1.4822863340377808, -1.2504158020019531, -3.0080623626708984, 0.08020848035812378, 1.3362430334091187, 0.9945034384727478, -1.0464504957199097, -0.993293821811676, 1.3973056077957153, -0.3018740117549896, -0.5238179564476013, -0.9385645389556885, -1.5025306940078735, 0.5506247878074646, -0.46305111050605774, -0.15745243430137634, 1.465593934059143, -1.3307363986968994, 0.3454725444316864, -0.5886766910552979, 1.3655935525894165, 1.2040605545043945, 0.4104326665401459, 0.9642510414123535, -0.05930035188794136, -0.2892666459083557, 0.6722937226295471, 0.24992933869361877, -0.8541764616966248, 1.087571382522583, 0.31212958693504333, -1.148212194442749, 0.4578723609447479, 0.992467999458313, -0.45186281204223633, 0.8881400227546692, 0.6778038740158081, 0.05187598243355751, 0.6125964522361755, -0.9891208410263062, 1.483473300933838, 1.0839358568191528, 0.10227622091770172, -1.3798632621765137, 2.0629048347473145, -0.45692676305770874, 0.869970977306366, 1.3845149278640747, -0.40587887167930603, 0.3213057219982147, -0.136869415640831, 2.5751819610595703, -0.11907970160245895, 0.9509357213973999, -0.7388575673103333, -0.7832666635513306, -0.24385035037994385, -1.7808234691619873, -0.8945289850234985, 0.16307653486728668, 0.8042925596237183, -1.7845107316970825, -0.06012794002890587, 0.7437846660614014, -1.3690314292907715, 0.1781909465789795, 1.9536043405532837, -1.1021044254302979, 0.20029683411121368, -2.7455332279205322, -0.8536719083786011, 0.027548007667064667, -0.4424656927585602, 0.47559210658073425, 0.004214470274746418, -2.0003204345703125, 0.9259505867958069, 0.8876620531082153, -1.3966354131698608, -1.3703292608261108, 3.035508632659912, 0.6080578565597534, -0.3164246380329132, -0.8661483526229858, -0.4568745493888855, -0.012607114389538765, -1.2356367111206055, -0.09390649944543839, 0.11859214305877686, 0.5562522411346436, -0.28774815797805786, 0.29685690999031067, -0.4435892701148987, 0.09159186482429504, 0.9472330808639526, 1.1296322345733643, 0.2761155962944031, 0.968755841255188, -2.1967897415161133, -1.9103230237960815, 0.7505840063095093, 1.068947672843933, 1.3149709701538086, 1.480918049812317, 1.1420031785964966, 2.036552667617798, 0.4794350862503052, 0.9655022025108337, -0.30592605471611023, 1.8932762145996094, -1.6886554956436157, 1.0047104358673096, -1.1875351667404175, 0.4792679250240326, -0.38302087783813477, 1.3546159267425537, 0.969854474067688, -0.4192740321159363, -0.9577213525772095, 1.7106420993804932, 0.2201612889766693, -2.333976984024048, 1.4928898811340332, -0.3497925400733948, -0.5827147960662842, -0.3627339005470276, -0.05702875182032585, -1.5578612089157104, 1.8021718263626099, -0.8475052118301392, 0.9058881998062134, 0.5852904319763184, 0.27073344588279724, 0.7009874582290649, 0.7081738710403442, -1.7128736972808838, -0.28067582845687866, 0.20171037316322327, -0.7387917637825012, 0.1335678994655609, -0.31011757254600525, -0.8250755071640015, -0.9236660003662109, 0.5066764950752258, 2.6673953533172607, -2.175285816192627, 0.056882381439208984, -0.37515920400619507, -0.16888581216335297, -1.4729372262954712, 2.003917932510376, 0.42277616262435913, 0.5850504040718079, 1.253649353981018, -0.874168336391449, 0.8172633647918701, -0.044112496078014374, 0.21904727816581726, -2.6256473064422607, 0.10749601572751999, 0.576010525226593, -1.39983069896698, 0.3670085072517395, -0.40865030884742737, -0.7502533793449402, 0.19847826659679413, -0.11618237942457199, 0.5579620599746704, -1.111583948135376, 0.33785250782966614, 0.8360733389854431, -1.6025017499923706, 2.2087013721466064, 0.4971912205219269, -1.2260921001434326, -1.6249076128005981, 1.1165307760238647, -2.0313243865966797, -1.2024635076522827, -0.5996824502944946, -0.6689531207084656, 0.8838707804679871, -0.30990591645240784, 0.4381350874900818, 0.8268446922302246, -0.8184539675712585, -0.5322690010070801, -1.223278522491455, 0.31005242466926575, 0.7538687586784363, 0.00898717064410448, 1.3246219158172607, -2.0954370498657227, 0.4982532858848572, 0.301088809967041, -0.897708535194397, 1.8568147420883179, -2.0137033462524414, 0.8381434082984924, 2.1917214393615723, -0.14140012860298157, -0.8340297937393188, -0.3728696405887604, 0.2691914737224579, 0.5728265643119812, -1.8388268947601318, 0.4371180832386017, 0.7860438823699951, -0.8514483571052551, -1.526464581489563, 2.1756644248962402, 0.1704634130001068, 0.2660920023918152, -1.1181035041809082, -0.21989788115024567, -0.47805091738700867, -0.7758585214614868, 1.0974899530410767, 0.42755770683288574, 1.296307921409607, -0.3633449375629425, 0.5818949937820435, -0.4848061203956604, 0.9388136267662048, -1.4647372961044312, -0.3527475893497467, -0.19523493945598602, -0.5598914623260498, 1.126500129699707, -0.7698985934257507, -1.144593596458435, -0.5037074685096741, -1.1076507568359375, -0.5404162406921387, 2.0099003314971924, 1.8536319732666016, -0.7249884009361267, -0.6359518766403198, -1.238701581954956, 0.6565963625907898, 1.5013092756271362, -1.7381329536437988, -0.9595353603363037, -0.9801300168037415, 0.7531777620315552, 0.06429805606603622, 1.4775052070617676, 0.6596943736076355, -2.2473907470703125, -1.7622294425964355, 1.0472074747085571, 0.11608400195837021, 0.29994162917137146, -0.11906523257493973, -0.5140546560287476, -0.869188666343689, -0.6829630136489868, -0.581631064414978, 0.5238742828369141, -0.8053123354911804, -0.7418773770332336, -0.75473952293396, 0.9025319814682007, 0.0556333102285862, -0.2948085367679596, 0.44379284977912903, -0.10105127841234207, -1.1054191589355469, -0.6722247004508972, -0.12001814693212509, -0.9787994027137756, 0.14236316084861755, 0.27424371242523193, -1.037326693534851, 0.27233827114105225, 0.5206270813941956, -0.6675575375556946, -0.3088721036911011, 1.9798911809921265, -0.3419780731201172, -0.5597701072692871, 0.7655907273292542, 1.3325371742248535, -0.6274042129516602, -0.47034621238708496, -1.8323619365692139, -1.5171047449111938, 0.28130224347114563, -1.0282294750213623, 1.4603476524353027, 0.20871075987815857, 0.8094465136528015, -0.7615590691566467, 0.6586592793464661, 1.0872496366500854, -0.9967100024223328, 0.5401337146759033, -1.4142634868621826, -0.9390213489532471, -1.4023460149765015, 1.2855486869812012, 0.7703972458839417, -0.14040742814540863, -0.7122035622596741, 0.3247164487838745, 0.8754634261131287, -1.8276885747909546, -0.16612902283668518, -1.5189074277877808, 0.5334981083869934, 1.0968329906463623, -1.5542800426483154, -0.5004981756210327, 1.8263802528381348, 0.6821996569633484, -1.3629952669143677, 0.6464113593101501, -0.32797667384147644, -0.7799500226974487, 0.24278651177883148, 0.46075019240379333, -0.39848393201828003, 0.15549281239509583, 0.2303025722503662, -0.4433780312538147, -1.3831477165222168, -1.6111599206924438, 1.235858678817749, -1.1673967838287354, 0.6426100730895996, 0.951077938079834, -0.04093209654092789, -0.21926185488700867, -1.7789181470870972, 1.459184169769287, -0.19902662932872772, 0.6166452169418335, 1.319312334060669, -0.41860824823379517, -0.7221814393997192, 0.9299420118331909, 0.4717261791229248, -1.218706488609314, 0.7959625720977783, 0.7147934436798096, -0.18899890780448914, 0.2522921562194824, -1.0769766569137573, 0.36042165756225586, 1.3454015254974365, -0.3173743486404419, 0.32491686940193176, -0.7168828845024109, -0.2411358803510666, -0.7275200486183167, 0.18237322568893433, -0.27958211302757263, 1.0335739850997925, 2.4335951805114746, -1.1461470127105713, -1.0500448942184448, -0.5988523960113525, -0.5481000542640686, -0.13055917620658875, -0.8350531458854675, -0.3216229975223541, 1.1741092205047607, -0.46475914120674133, 1.060415506362915, 1.0689119100570679, 1.728267788887024, 0.7329810857772827, -1.5475223064422607, -1.268049955368042, 0.3372976779937744, -0.3301815092563629, 1.8045529127120972, 1.3572523593902588, 0.847916841506958, 0.16770291328430176, 0.04231121391057968, -0.7393687963485718, 1.308752179145813, -0.8161695599555969, 0.11326447129249573, 0.5520650148391724, -1.4670817852020264, 1.3273423910140991, 0.26508384943008423, -0.21034753322601318, -1.9473400115966797, 1.1424174308776855, -0.6536087393760681, -0.6687490344047546, 0.44547367095947266, -0.4899600148200989, 0.6675553321838379, 1.0276968479156494, -0.4468592405319214, -1.9048348665237427, 0.9603680372238159, -2.182957172393799, 0.9709979891777039, -1.6969106197357178, -0.7640986442565918, 0.16288766264915466, -0.2113995999097824, 1.037885308265686, 0.8991037607192993, -0.9831323623657227, -0.46171268820762634, -1.3095510005950928, -1.1915580034255981, -0.0558125302195549, -0.4289836585521698, 0.8560228943824768, 1.7514795064926147, -0.8019644021987915, 0.05350550636649132, 0.9300333261489868, -0.28425905108451843, -0.1629713922739029, -0.6266154050827026, 1.1284552812576294, -0.6202711462974548, -0.9571232199668884, -0.6779753565788269, -0.24685357511043549, 0.7046751976013184, -0.8676157593727112, 0.7441712021827698, -0.1812821477651596, -0.9707846641540527, -0.9320753216743469, 1.1085412502288818, 0.9602193832397461, 0.7716919779777527, 1.3052960634231567, 0.5097031593322754, -0.8733928799629211, 0.04385386407375336, 0.14067263901233673, -1.2955131530761719, 0.07036581635475159, -0.5378481149673462, -0.43049156665802, -0.06488346308469772, -0.8881248235702515, -2.006934404373169, -0.5088836550712585, -1.1183040142059326, 1.4279216527938843, 2.0605711936950684, 1.4114662408828735, 0.20564624667167664, -0.9069571495056152, 0.6673871278762817, 0.9218336939811707, 0.16827291250228882, 0.8203009366989136, -0.12855438888072968, 0.3680202066898346, 0.31620609760284424, -0.510799765586853, 0.5514979362487793, 0.832715630531311, 0.4146701991558075, -0.13974246382713318, 2.0607340335845947, 0.8698869347572327, -1.7359287738800049, -2.19515323638916, -0.6811532378196716, 0.5353863835334778, 1.1923863887786865, -1.5174403190612793, 1.932060956954956, -0.242447629570961, -0.2512904703617096, -0.29473263025283813, 0.13639302551746368, -0.7382024526596069, 0.7979302406311035, 1.0773119926452637, -0.2864258885383606, 1.0116161108016968, 0.6990624666213989, 0.16447056829929352, -1.550477147102356, 0.9988552927970886, 0.45064055919647217, 0.3062070608139038, -1.1312659978866577, -0.4878659248352051, -1.3319960832595825, -0.13054445385932922, 0.2878497540950775, -0.19746820628643036, -1.3991715908050537, -0.3436868488788605, 0.3669525682926178, 0.8516970276832581, 0.9238263368606567, -0.6377846598625183, -1.1993118524551392, -0.9563584923744202, 0.21431462466716766, 0.6784161925315857, 0.7447441220283508, -0.3526272177696228, -0.6474695205688477, -0.4613305628299713, -0.17570409178733826, 0.9284995794296265, -1.3333548307418823, 0.3451390564441681, -0.13068002462387085, -0.08944158256053925, -0.10562432557344437, -0.3385298550128937, 0.34760582447052, -0.6186500191688538, -1.2929612398147583, -1.135294795036316, 2.2731971740722656, 1.6812143325805664, -1.516194462776184, 1.0673633813858032, 0.3044053912162781, -0.6368023157119751, -0.07825513929128647, -1.0635069608688354, 0.9896788001060486, 1.5141314268112183, -1.8144710063934326, -0.39962053298950195, 1.2749122381210327, -0.32837486267089844, -0.013953489251434803, -0.08922756463289261, 0.5199138522148132, -0.18644332885742188, 0.05317828804254532, 0.45998451113700867, -0.29618996381759644, 0.8924388885498047, -0.935222864151001, -0.934613049030304, 0.6458026766777039, -0.11390736699104309, -0.6543508172035217, 0.531975269317627, -0.039491355419158936, -0.49952784180641174, 0.7945717573165894, -1.9722386598587036, 1.790135145187378, -0.2891433537006378, -0.693280041217804, 0.7898539900779724, -0.41931214928627014, -0.30051711201667786, 2.496837854385376, -1.764710783958435, 0.5573750734329224, 1.04045832157135, -0.5449045896530151, -0.272888720035553, -0.6273051500320435, 1.027898907661438, -0.3953537046909332, -0.9751326441764832, -1.219385027885437, 0.02517038956284523, -1.461541771888733, -0.43587982654571533, -0.508503258228302, -0.0761893093585968, -1.0077718496322632, -1.0014022588729858, 0.9735099077224731, -1.8830031156539917, 0.7463252544403076, 0.34697428345680237, 0.44583702087402344, -1.2992103099822998, -0.8805890679359436, -0.9780604243278503, -0.30501294136047363, 0.8905296921730042, -1.0695003271102905, -1.294296383857727, 0.8164077401161194, -0.6412109732627869, -0.6024051904678345, -0.07029605656862259, 1.1681463718414307, -0.030208054929971695, -1.4855358600616455, -1.6941567659378052, -0.8732016086578369, -0.3869064450263977, -0.7481929659843445, 1.2139149904251099, -0.8865705132484436, -0.16794070601463318, 2.2536821365356445, -0.6506617069244385, 0.5086138844490051, -1.24366295337677, -0.30770987272262573, 0.11653682589530945, 1.2066130638122559, -1.6836397647857666, -0.9598308205604553, -1.6357557773590088, 2.330948829650879, -0.5356097221374512, -0.4936140179634094, 0.25881659984588623, -0.4153934419155121, 0.023287726566195488, -0.3374197483062744, -0.31439846754074097, -0.12563937902450562, 2.056629180908203, 0.7566088438034058, -0.6003556847572327, 0.016617080196738243, -2.7261781692504883, -0.21781843900680542, -1.2717615365982056, -0.787284791469574, -0.2907807528972626, -0.48130783438682556, -0.08243376761674881, 0.027933955192565918, -0.5417847633361816, 0.17503122985363007, -1.6018171310424805, 1.3121486902236938, 0.10187237709760666, 0.12805549800395966, 0.9000412225723267, -1.474454641342163, -0.6787137389183044, -0.8050196170806885, -1.022696614265442, -2.115755081176758, -1.5513404607772827, 0.18111354112625122, 0.2976606488227844, -0.6082161664962769, 0.1163727417588234, 0.11513077467679977, -0.1505468636751175, 0.5636613965034485, 0.4640749394893646, 0.45597952604293823, -0.8051327466964722, 0.7721586227416992, 1.474810242652893, 0.7826043367385864, 1.1330389976501465, 0.08820047229528427, 0.4219215512275696, -0.020477866753935814, -1.0527209043502808, 0.9829984903335571, -0.8255849480628967, 1.43723726272583, -2.3561182022094727, 0.08718692511320114, 0.07618670165538788, -1.1802890300750732, 0.2146638035774231, -0.8650203347206116, -0.02344716340303421, 0.4274628460407257, -0.32405710220336914, -0.08353646099567413, -0.18832874298095703, -0.8476464152336121, 0.25726598501205444, -0.6913368701934814, -0.34841635823249817, -0.04245636984705925, -0.43021827936172485, 0.5020917654037476, -0.49933168292045593, 1.9341866970062256, 0.8719215989112854, -0.547825813293457, 0.6044467091560364, 1.6947057247161865, -0.35668885707855225, 0.030126137658953667, 0.4063602387905121, -0.6578388810157776, -0.496639221906662, -0.7841949462890625, 1.3201196193695068, 0.84941166639328, 0.23086801171302795, -1.0692495107650757, 0.3712116777896881, 0.8818187713623047, 1.2357120513916016, -0.14344726502895355, -0.33621278405189514, 2.5977227687835693, 1.538559913635254, 1.1297062635421753, -0.8231091499328613, -0.5874446034431458, -0.44652116298675537, -0.996095597743988, -1.2771413326263428, -0.756832480430603, -1.3966959714889526, -0.5015305280685425, 0.036759600043296814, 0.16660277545452118, 1.4888149499893188, 0.23846624791622162, 0.8145387768745422, 0.9324955344200134, 1.9533379077911377, -1.1063071489334106, 0.33212798833847046, 0.5149442553520203, 0.4383562505245209, -0.526249349117279, 1.4132782220840454, -0.8253027200698853, 0.21472562849521637, 0.13931716978549957, 0.7317931056022644, -0.8930942416191101, 0.6085609197616577, 2.6829543113708496, 0.15188662707805634, 0.1967640072107315, 1.0837328433990479, 1.2944371700286865, -1.2845653295516968, -1.9341579675674438, 0.7708895802497864, 1.9215906858444214, -1.8572592735290527, 0.9411174058914185, -0.35801878571510315, 1.685472846031189, -1.2010136842727661, 0.7360018491744995, 0.8525124192237854, 1.042162537574768, 1.0494260787963867, -0.559964656829834, 1.1059401035308838, 1.1349329948425293, 1.6701381206512451, 0.5445297360420227, 0.7124488353729248, -0.9875471591949463, 0.08414532244205475, 0.8370164036750793, 0.1570470780134201, 0.3861334025859833, 0.6314045786857605, -1.2624785900115967, -1.8213770389556885, 0.2357717901468277, 1.2462221384048462, -0.5939174294471741, -0.8988776206970215, 0.36904793977737427, -0.8273501396179199, -0.10556124895811081, 0.3211962580680847, -0.1897972822189331, 1.4389748573303223, 0.42409995198249817, 1.0068299770355225, 0.4001508951187134, 0.3626261055469513, -0.2568456828594208, -0.2676678001880646, 1.3544162511825562, -0.5352263450622559, 3.4246010780334473, 0.7624326944351196, -0.7623223662376404, -0.55646812915802, -0.7792588472366333, -0.618815004825592, -0.19363556802272797, 0.4487205743789673, 0.7828672528266907, -1.1574852466583252, -0.16155502200126648, -2.147519111633301, -0.7944661974906921, 0.015502102673053741, 0.9029114842414856, -0.6089446544647217, -0.315452516078949, 1.4871060848236084, -0.22246153652668, -2.0330681800842285, 1.0572552680969238, 0.5899580717086792, -0.7128863334655762, -0.19183221459388733, -0.6092045903205872, 0.007462969049811363, 1.9747027158737183, -0.5734211802482605, -1.4269542694091797, -0.5787872672080994, 0.7657925486564636, 0.6939340829849243, 0.08436102420091629, -0.12496735900640488, 1.440858244895935, 0.3444584608078003, 0.7062907814979553, 1.13838791847229, 0.8610071539878845, -0.11198526620864868, -3.1191165447235107, 0.18329742550849915, -0.2666706442832947, 0.7616953253746033, -0.20934098958969116, 0.21797911822795868, -0.5533015727996826, 1.645619511604309, -0.49146413803100586, -1.7644240856170654, 1.6657878160476685, 0.9106566905975342, -0.6124802827835083, 0.8036726117134094, -0.1370796412229538, 1.967598557472229, -0.1978994458913803, -0.7445363998413086, -0.984637975692749, 1.1849241256713867, -0.4483274221420288, -0.7070645689964294, -0.9186245799064636, 0.22539383172988892, -0.7308365702629089, -0.12224896252155304, 1.481972336769104, 0.3319530189037323, -2.840031385421753, -0.19714631140232086, 2.090031147003174, -0.5896705985069275, 0.8268929719924927, -2.117323637008667, -0.007942447438836098, -0.23013249039649963, -0.15018758177757263, -0.44505396485328674, -1.4913413524627686, 0.08227404206991196, -0.6581897735595703, -0.23875188827514648, 1.1636704206466675, 0.3405002951622009, -0.44060543179512024, -0.28113579750061035, -1.2023342847824097, -1.5833475589752197, 0.6482928395271301, 0.7179329991340637, 1.1620038747787476, -1.159886121749878, 0.3781700134277344, 0.2627370357513428, -0.579466700553894, -0.9929773807525635, 1.808715581893921, -0.10127705335617065, -0.17785793542861938, -0.3672747015953064, 1.1143395900726318, 0.5774562954902649, -2.2699320316314697, 0.41766658425331116, -0.6353654265403748, 0.18903298676013947, 0.2831439971923828, 0.5403077602386475, 1.6067506074905396, -0.632535994052887, -1.7002862691879272, 0.11884156614542007, 0.42041105031967163, 0.32536861300468445, 1.154000997543335, -1.831955909729004, -0.8119171261787415, -0.15504342317581177, 0.3977862298488617, 3.3202779293060303, -0.13832275569438934, -0.066858671605587, -0.546516478061676, 1.2685989141464233, 0.4629104435443878, 0.6702561974525452, -1.3433775901794434, -0.3835637867450714, -0.6803893446922302, 0.7890806198120117, 1.8255603313446045, -0.24953585863113403, 1.5627915859222412, 0.1051463633775711, 1.4550737142562866, -0.44388264417648315, -1.1423609256744385, -1.6099039316177368, -0.6177328824996948, 1.6060330867767334, 1.0132384300231934, -0.5785007476806641, -0.10142362862825394, -0.2865441143512726, 1.409614086151123, 1.6858981847763062, -0.4773828089237213, -0.44858917593955994, 0.26710835099220276, -0.3823172152042389, 0.8023268580436707, 0.363996684551239, 1.3167108297348022, -0.056474991142749786, -0.05898221954703331, 1.6644837856292725, 1.3823316097259521, 0.29990488290786743, -1.0129363536834717, -2.2393527030944824, 1.258844256401062, -0.1762954145669937, -1.5969955921173096, 0.10281769931316376, 0.6397461295127869, 0.20664343237876892, 0.9546970725059509, 1.0619248151779175, 1.4885941743850708, -0.35332366824150085, 0.08374649286270142, 0.5562774538993835, -1.947704792022705, 0.31539347767829895, -1.7839982509613037, 0.6183996200561523, -1.4514390230178833, -1.3436065912246704, 1.1791540384292603, 1.017532467842102, -0.6746129989624023, 0.6817469596862793, -0.6277776956558228, -0.2245142161846161, -0.06334354728460312, 0.05169250816106796, -0.04120826721191406, 2.115401268005371, 0.4904320538043976, -1.4976186752319336, -1.1763291358947754, 0.1885869801044464, 4.153605937957764, 1.145743489265442, 3.369513988494873, 0.21403026580810547, 0.7577606439590454, 1.1262869834899902, -0.8128470182418823, -0.807612419128418, 0.5802505016326904, -0.2364928424358368, -0.21608467400074005, -1.2629637718200684, -0.4391292929649353, -1.516786813735962, 0.7064166069030762, -0.7969030141830444, -0.06763844937086105, -1.3335908651351929, -1.3756096363067627, -1.249103307723999, -0.5814249515533447, -0.8932482600212097, 0.5311964750289917, 1.2527488470077515, -0.342304527759552, 0.2641436457633972, 0.340806245803833, -0.16703316569328308, -0.6018229126930237, 1.3025882244110107, -0.14223283529281616, -1.1151982545852661], [-0.8565415740013123, -0.06280972063541412, 0.9609566926956177, 0.2486891895532608, 0.019467197358608246, -1.928198218345642, -0.01804998330771923, -0.8206809163093567, -0.38588225841522217, -1.3042513132095337, 1.436960220336914, -0.27288204431533813, -1.7998554706573486, -0.0922800749540329, 0.660936713218689, 0.9840921759605408, 2.0141594409942627, 0.3019171357154846, -1.7957086563110352, 0.006154293660074472, 0.5190115571022034, 0.5255916714668274, -0.6006328463554382, -1.567825198173523, 0.7054138779640198, 0.7344467639923096, -1.5094250440597534, 0.9224309325218201, -1.6862293481826782, 0.8197010159492493, 0.1591680943965912, -0.4923989772796631, 0.49667540192604065, -0.6274015307426453, 0.11180131137371063, -1.1908329725265503, -0.1715867817401886, -0.36095571517944336, 0.20011046528816223, -0.22082845866680145, 2.048722743988037, -0.10690722614526749, -1.415911316871643, -0.7862831354141235, 0.1791171431541443, 0.10613556951284409, -1.397750735282898, 0.5867259502410889, -1.7917113304138184, -1.3733670711517334, -0.8660712242126465, 1.285569190979004, 1.9115592241287231, -0.23086272180080414, -2.06375789642334, -0.29086148738861084, -0.503780722618103, -1.7975367307662964, -1.0092480182647705, 0.620460569858551, 0.38278236985206604, -0.7105714678764343, 0.2688393294811249, 0.9554120302200317, 0.6886190176010132, -0.7590206861495972, -1.6932098865509033, 0.7075412273406982, -1.0082969665527344, -0.8491724133491516, -1.6496087312698364, 0.33333060145378113, -1.5756462812423706, -0.5433874726295471, -2.8837223052978516, -0.01393372192978859, -1.5266895294189453, -0.580921471118927, -1.6819450855255127, -1.0093387365341187, 0.4580787420272827, -1.2902578115463257, 0.4523453116416931, -0.04168212041258812, 1.7739012241363525, -0.9310686588287354, 0.03446704521775246, -0.7081984877586365, 0.682778537273407, -1.4182170629501343, -0.720064640045166, 0.20232491195201874, -0.7653356194496155, 0.4496859908103943, 1.7968964576721191, 1.2968450784683228, 2.2175259590148926, 0.3199566900730133, -0.2834220826625824, -1.2042049169540405, -1.7116490602493286, -0.7687291502952576, 0.9356275200843811, -0.5267597436904907, 1.4311542510986328, -1.0656903982162476, 0.16299405694007874, 0.7641147375106812, -0.0007697694818489254, -0.5765569806098938, 0.3940107226371765, -0.06798691302537918, 0.43827855587005615, 0.2627102732658386, -1.058044672012329, -0.0482151173055172, 1.4045673608779907, 1.2208404541015625, 0.10859595984220505, 1.2097917795181274, -0.1711207926273346, 0.27552086114883423, -1.83379328250885, 0.828046977519989, -1.2003434896469116, -0.407695472240448, 0.3233414590358734, -0.02942907251417637, 0.9644351601600647, -0.6955066323280334, -3.413790225982666, 0.8237837553024292, -2.5499672889709473, -0.6428773403167725, -0.15466003119945526, -0.18979017436504364, -0.6810157895088196, -0.6562496423721313, 0.2845313549041748, -0.8122125864028931, -0.2742707133293152, 1.0822975635528564, -0.4762532413005829, -1.446671962738037, 0.9507196545600891, 0.503180742263794, 1.6945563554763794, 0.01596992276608944, 1.0465357303619385, 0.5360040664672852, -0.9568719863891602, -1.326568365097046, -1.0413020849227905, 0.22109173238277435, 1.8107693195343018, 0.9382964968681335, 1.7933108806610107, -0.39115336537361145, 0.5109550952911377, -2.616023302078247, -0.143023282289505, 1.906089186668396, 0.12200656533241272, 0.48963335156440735, -0.2354700267314911, 0.6620074510574341, -0.6112061142921448, -0.5379887223243713, 1.3502999544143677, -0.03841566666960716, -1.0339068174362183, 0.003658358007669449, 0.21173736453056335, -2.105130195617676, -0.6619714498519897, 0.5420401692390442, 1.926974892616272, -0.3158142566680908, 1.560422420501709, -0.4054873585700989, -0.34381306171417236, -0.7021080255508423, -1.7944186925888062, -1.949020504951477, 0.9915174841880798, 0.5274677276611328, 0.2653917074203491, -0.5376520752906799, -1.4673265218734741, -0.4221455454826355, -1.554091215133667, -0.2940986752510071, -0.8835718631744385, 1.2774109840393066, -0.405540406703949, 1.0948314666748047, 1.1388431787490845, 0.5709869861602783, -0.07592211663722992, 1.1400504112243652, 1.6868952512741089, -0.05505738779902458, -0.7856601476669312, 0.1822398453950882, -0.5454006195068359, -0.07546563446521759, 0.36347293853759766, -0.4350300431251526, -0.7003768682479858, 0.634016215801239, 0.31104907393455505, 1.8872982263565063, 0.3779008984565735, 0.4283735752105713, -0.6926332712173462, 0.03534264117479324, 0.9735052585601807, 0.4161945581436157, -0.9617112278938293, 0.33165988326072693, 0.011508875526487827, 0.0722481980919838, 1.226879596710205, 1.155440092086792, -0.6772513389587402, -0.9719247221946716, 0.38410964608192444, -1.8578112125396729, 0.6291696429252625, 0.5262759327888489, 0.14433500170707703, -0.820277988910675, -2.142582416534424, 0.14614200592041016, 0.5157764554023743, -0.5285080671310425, -0.33233118057250977, 0.6805139183998108, -0.5410328507423401, -0.16584309935569763, 1.0367319583892822, -0.831514835357666, -0.42097553610801697, -0.32541224360466003, -0.316992849111557, -0.521094560623169, -1.4525939226150513, -0.8142873644828796, -1.0197111368179321, -0.8362854719161987, 0.8097786903381348, -1.6180678606033325, 0.5472522377967834, -0.4994196593761444, -1.083317518234253, 0.5591368079185486, 0.31411147117614746, 0.6055303812026978, -2.724963665008545, -1.4095629453659058, -0.06195017695426941, -0.7908485531806946, -0.5608890056610107, -1.5299360752105713, 1.0676636695861816, -0.11642202734947205, -2.494986057281494, 1.2901268005371094, 0.49716201424598694, -0.26172539591789246, 1.1156688928604126, -0.8790517449378967, 0.25723016262054443, -2.1229324340820312, 0.3692474365234375, -0.6321272253990173, -1.6327316761016846, -0.4612441956996918, 0.12816379964351654, 0.8168991804122925, -1.3738017082214355, -1.539338231086731, -0.8193615078926086, -0.5053943991661072, 1.0608561038970947, -1.6348544359207153, 0.5535632371902466, 0.2998650074005127, -0.45149514079093933, 1.2660951614379883, -0.5621671676635742, 0.25552958250045776, -0.2287975251674652, -0.08790082484483719, -1.6199685335159302, -0.4827713966369629, -0.9349079728126526, 0.7672315835952759, 0.5249245762825012, 0.9376519322395325, -0.1784675419330597, 1.3015080690383911, 0.13465991616249084, -1.21238374710083, 1.0226709842681885, 1.987259030342102, -0.892525315284729, -1.1289335489273071, 1.1374486684799194, 1.4477936029434204, 1.3958797454833984, 0.21577465534210205, 0.3996273875236511, 0.6203290224075317, -2.246718168258667, 0.518254816532135, 2.2853450775146484, 0.09246870130300522, 0.6558890342712402, 0.7107540965080261, 0.9558560848236084, 0.9230498671531677, 0.13635572791099548, -0.17480313777923584, 1.3442844152450562, -1.0288269519805908, -0.2421116828918457, -1.0319308042526245, -0.1946306675672531, 1.6106126308441162, 0.5678519010543823, -0.663249135017395, -0.20307594537734985, -0.7984583377838135, -0.01305437833070755, 0.9309132695198059, 1.0920237302780151, 2.963226795196533, -0.8925728797912598, -1.491969108581543, -2.391832113265991, -0.4220118522644043, -0.5665401816368103, 0.8791048526763916, 0.6471562385559082, -0.08587908744812012, 0.06302900612354279, 0.7056446671485901, -0.4797598719596863, -1.4178380966186523, -1.2880593538284302, -0.6148738861083984, -1.0682963132858276, 0.22142435610294342, -0.9764372706413269, -0.9627629518508911, -0.08551087230443954, 0.5784028768539429, 0.2669644057750702, 1.0203309059143066, 0.6438953876495361, -1.2852460145950317, 0.09011827409267426, 0.29107409715652466, -0.6175351142883301, 0.28028926253318787, -2.2210237979888916, -0.6825046539306641, 1.711201786994934, -0.09114480018615723, -1.3372972011566162, 0.05729944258928299, 0.7900159955024719, -1.4333277940750122, -0.4658396542072296, -0.18664342164993286, -0.7732129096984863, 2.030320405960083, -0.9732207655906677, -0.05655688792467117, 0.27486470341682434, 0.16267737746238708, -0.5811820030212402, -2.0280988216400146, -0.7603943943977356, -0.6212934851646423, 1.561155915260315, 0.44167137145996094, 1.3592158555984497, -0.7104502320289612, -0.0172099769115448, -0.7823649644851685, 0.6423859000205994, -0.047197841107845306, -0.9849516153335571, 0.06991147994995117, 0.8330227136611938, -0.29184502363204956, -0.22631052136421204, 0.5006550550460815, 0.5779033303260803, 1.1411134004592896, 1.4365367889404297, 0.4922817647457123, -0.9197050333023071, 0.30641987919807434, -0.9936051368713379, -0.6013152599334717, -2.0747082233428955, 0.16108447313308716, 1.2237056493759155, 1.0150150060653687, -0.0944056361913681, -0.33189529180526733, 0.07918432354927063, -0.2584993541240692, -1.3589152097702026, 0.20231013000011444, -1.0685172080993652, -0.9370635747909546, 1.9583320617675781, 0.8932024240493774, 0.008063716813921928, -0.7730008363723755, -1.4491688013076782, -0.10650315880775452, 2.468554735183716, -0.24662071466445923, -0.004868870135396719, 0.10009336471557617, 0.899093508720398, -1.1724556684494019, 0.027826856821775436, -0.9414917230606079, -0.6086241006851196, -1.707332968711853, 1.2181497812271118, 0.8856178522109985, 1.4306976795196533, -0.6474457383155823, 0.27691835165023804, -2.234855890274048, 0.8772122859954834, 0.04715779051184654, 1.2219023704528809, 0.40674084424972534, -0.9428948163986206, -0.22202692925930023, 1.4854460954666138, -0.5527928471565247, 0.3538462817668915, -0.7298392057418823, -0.15956206619739532, -0.4829096496105194, 0.15051592886447906, 1.21343195438385, -0.8710210919380188, 0.32001930475234985, 0.5050414204597473, -0.668586015701294, -0.8080446720123291, 0.10220248252153397, 1.0657336711883545, -0.39673757553100586, -1.1232866048812866, -1.6170494556427002, -0.9077333211898804, 0.5522458553314209, 1.7428524494171143, 0.6333576440811157, -1.2968857288360596, 0.376099556684494, -0.7074663043022156, -0.8092244267463684, -0.45707201957702637, -1.5353672504425049, 0.9802835583686829, 0.4690377712249756, 1.7745630741119385, 0.06191462650895119, -0.06897931545972824, -1.5525848865509033, -1.5009901523590088, 1.1112252473831177, 0.2763378918170929, -0.516862690448761, -0.9764605164527893, -0.811029314994812, 1.349271535873413, 0.009348894469439983, 1.188901662826538, 0.2317379266023636, 0.3580167889595032, -0.29317888617515564, 0.534770131111145, -0.5407974720001221, -0.7932679653167725, -0.2982232868671417, 0.5087873339653015, 1.1822700500488281, -0.06643620133399963, 1.4797354936599731, 0.7450307011604309, 0.4388463497161865, 0.07981705665588379, -0.24856354296207428, -0.3603168725967407, -0.5563113689422607, 0.8106740117073059, 0.3666097819805145, -0.22187289595603943, -0.5407777428627014, -1.9288430213928223, -0.5343244671821594, 0.7362861633300781, -0.6642137765884399, -0.7509979009628296, 1.1592204570770264, 2.161911725997925, -1.054582118988037, 0.011974147520959377, -0.1953253298997879, 0.176859512925148, 1.506965160369873, -0.8230950236320496, 1.8559657335281372, -0.15227162837982178, -2.269603729248047, 0.8996604084968567, -0.6671486496925354, -0.20489588379859924, 0.2813662886619568, 1.2053076028823853, 0.8329404592514038, 0.29204973578453064, -1.1020296812057495, 0.8921198844909668, -1.3693934679031372, -0.5901352167129517, -0.19144485890865326, 0.8841867446899414, 0.3313904106616974, -0.20823228359222412, 1.5981075763702393, -0.0792776569724083, -0.1963285356760025, 0.07584207504987717, -0.8369544148445129, 0.527877151966095, -0.39127275347709656, -0.8187415599822998, -1.8166429996490479, 0.7087400555610657, -0.3959064781665802, -1.138749361038208, 0.32217487692832947, 0.2857852578163147, 0.14370280504226685, 0.692996621131897, -0.02740655466914177, -1.2390068769454956, 2.3228366374969482, -1.7848174571990967, 0.5890151262283325, 0.06758440285921097, -0.310909241437912, -1.2674401998519897, 0.2909943461418152, -0.15712690353393555, 0.8497201204299927, -0.3272411823272705, -0.7782249450683594, 0.5794791579246521, 0.7873448729515076, 0.04431440308690071, -0.1349346935749054, -0.9933369755744934, 1.1023812294006348, 1.472038745880127, -1.5607346296310425, -0.7025545835494995, 0.9211879372596741, 0.9062117338180542, -0.640735387802124, -0.3734600245952606, 0.051142700016498566, 0.4772869050502777, 0.6946590542793274, 0.24489383399486542, 0.2826543152332306, 0.867571234703064, -0.46972718834877014, 1.3141777515411377, -1.008931040763855, 2.1180508136749268, -0.48516324162483215, -0.932048499584198, 0.7388362288475037, 0.8762289881706238, 0.3791286051273346, -0.3632774353027344, -1.3138731718063354, 0.2501862943172455, 1.3793320655822754, -0.7043601274490356, 0.47213295102119446, -1.2061794996261597, -0.42074570059776306, 0.24080388247966766, -1.021894097328186, -0.5175021290779114, -0.758642315864563, -1.0267640352249146, 0.6400600075721741, -0.22658254206180573, -1.3141790628433228, -0.5396468043327332, 0.8080032467842102, -0.7213054299354553, 1.1663618087768555, 0.9795026779174805, 1.8897658586502075, -0.4558109641075134, 0.018746204674243927, 1.2037009000778198, -0.1830701380968094, -0.9647514224052429, 0.913172721862793, 0.2699759304523468, -0.2344292849302292, 0.6997663974761963, -0.4414732754230499, -0.2592885494232178, 1.4080474376678467, 0.040998686105012894, -0.412197083234787, -0.2784264087677002, -0.05773399397730827, -0.14876769483089447, 2.0372567176818848, 0.8075354695320129, 0.9101529121398926, 1.128130316734314, 0.3295205533504486, 1.2226200103759766, 2.138660430908203, -0.778527021408081, 0.3273467421531677, 0.41167107224464417, 0.8149661421775818, 0.23252244293689728, -1.2132530212402344, 0.5712800621986389, 3.350273847579956, 1.6102629899978638, -0.036901235580444336, -1.3465871810913086, -0.06326103210449219, -0.19051021337509155, 1.0285440683364868, -0.5717999935150146, -1.511664628982544, -1.0968791246414185, 2.745272636413574, -0.20779061317443848, 0.8971673846244812, 0.13154278695583344, -1.9513294696807861, 2.5488553047180176, -0.910758912563324, 0.4028420150279999, -0.3127228319644928, 1.7387055158615112, -1.015173316001892, 0.908393383026123, 1.1078131198883057, -0.0508652925491333, 0.3079364001750946, 1.4296396970748901, 1.4029967784881592, 1.070887804031372, 0.5107053518295288, -1.0079222917556763, -1.430332064628601, -2.4530985355377197, 1.6749262809753418, 1.1469801664352417, -0.5594202876091003, -0.05181380361318588, 0.34414002299308777, -0.4650494158267975, -0.23640689253807068, -0.3931429386138916, -0.36747756600379944, 0.08517586439847946, -1.3108229637145996, 0.8961703181266785, 0.5945685505867004, -0.5317874550819397, 0.5759923458099365, -0.7103644013404846, -0.053064338862895966, -0.5381544232368469, -1.1459733247756958, -0.5924004912376404, -0.14283306896686554, 0.721430242061615, -0.43614697456359863, 0.38056108355522156, 0.8018099069595337, -1.0492632389068604, 1.6572328805923462, 0.825350821018219, 0.31980812549591064, 0.8344602584838867, -0.6636895537376404, -0.969248354434967, 1.0297353267669678, -0.18728002905845642, 0.9230038523674011, -0.11008331179618835, 0.10347545146942139, 0.126331627368927, 1.4781601428985596, -1.3384851217269897, -1.1161017417907715, -1.1332777738571167, 0.35201141238212585, 0.3836299180984497, 0.9852520227432251, 0.8226460814476013, -0.35746002197265625, 0.3090144395828247, 0.6872768998146057, 1.1777187585830688, -0.5125043392181396, 1.306739330291748, -0.7050457000732422, -1.1575509309768677, 1.2277272939682007, -1.3069789409637451, -1.1866350173950195, -0.7722009420394897, -1.0498679876327515, 0.9690082669258118, -0.5947994589805603, -1.369903802871704, 0.9143585562705994, -0.9658135175704956, -0.5912853479385376, 1.0960948467254639, -0.3974553048610687, -2.013089656829834, -0.32786592841148376, -0.7241065502166748, -1.460126519203186, 1.180445671081543, -0.6515086889266968, -0.537299633026123, -0.7350817322731018, 0.3597627878189087, -0.3159415125846863, -2.6348884105682373, -0.05962095037102699, 0.28401899337768555, 0.0717848390340805, 1.01963210105896, 0.4769478738307953, -1.3968746662139893, -0.8308530449867249, 0.007257352117449045, -0.9648280143737793, -1.553059458732605, -0.04709288477897644, 0.49578768014907837, -0.1936412751674652, 0.5078650712966919, -0.25580382347106934, 0.404714971780777, 3.4604177474975586, 0.3480522036552429, -0.04434910789132118, -1.2250884771347046, 0.9568520188331604, 0.9501441717147827, 0.8246198296546936, 0.1688980758190155, 1.8704901933670044, 1.0232906341552734, 1.9581090211868286, 0.4577227532863617, -1.247918963432312, 0.10776721686124802, 0.13575656712055206, -0.6438257694244385, 0.6627699136734009, 2.2654573917388916, 0.49626222252845764, -0.16198180615901947, -0.3065111041069031, 0.2852271795272827, -0.7195248603820801, -0.6541495323181152, -1.6403350830078125, -2.0305721759796143, 0.5045021772384644, 0.0498681366443634, -0.1621655821800232, -0.009138417430222034, 0.5774446725845337, 0.386442095041275, 0.9716818928718567, 1.344606637954712, -0.4578511416912079, 0.8266913890838623, -0.18544211983680725, -1.3892362117767334, -0.29391950368881226, -0.5009154081344604, -1.2264817953109741, 0.6685965061187744, 0.4696924686431885, 0.6587408781051636, -0.8472470045089722, -1.0913859605789185, 1.8910928964614868, 1.5401962995529175, 0.5539140105247498, 1.3111183643341064, 0.18664264678955078, -0.6887460947036743, 0.21054129302501678, -0.9981600046157837, -1.0017180442810059, -0.8408884406089783, 0.93003249168396, 1.4082711935043335, 0.05781632661819458, 0.06295239925384521, -2.2057905197143555, 0.2229643166065216, -0.945406436920166, -0.4258972704410553, -0.5860692858695984, -0.015955574810504913, -1.258729338645935, 0.22894607484340668, 0.43430325388908386, -0.21706245839595795, 2.065217971801758, -0.1779288500547409, -0.19837896525859833, -0.8362256288528442, 0.0005191469099372625, -0.41924282908439636, 0.69279545545578, 0.3128558397293091, -1.1062581539154053, 0.18021796643733978, 0.5158476233482361, -0.2794357240200043, -0.5488370060920715, -0.8386653661727905, -0.34831055998802185, 1.0392615795135498, 0.13410945236682892, -0.4454076886177063, 0.8428893685340881, -2.374882936477661, 0.03111867606639862, 2.8740618228912354, -1.156457781791687, -1.3865193128585815, -0.5376006960868835, -1.3378163576126099, -0.36595162749290466, 1.9741564989089966, -2.215085983276367, 1.1843317747116089, 1.0260683298110962, -2.0074734687805176, -1.510634183883667, -0.1453079879283905, -1.5190659761428833, -0.7446159720420837, 0.3096192479133606, 0.6845174431800842, -0.7964833974838257, 0.3142072558403015, 1.2675647735595703, 0.06118011102080345, 0.5385764837265015, -0.8736979961395264, 1.0312254428863525, 0.049460481852293015, -0.8730036020278931, -1.7312098741531372, 1.1107161045074463, -1.1281119585037231, -0.7311372756958008, 0.5242243409156799, 0.9137625694274902, -0.7358048558235168, 1.37723970413208, 0.5824841260910034, -2.081082582473755, -0.7610284090042114, -1.2016135454177856, -2.5873897075653076, -1.5349026918411255, 0.07984136044979095, 0.8122953176498413, -0.0712941586971283, -2.107832431793213, 0.2003890872001648, 0.05265497416257858, -0.9080373048782349, 0.006287450436502695, -0.432907372713089, 0.8430183529853821, -1.2875497341156006, -0.49395886063575745, 1.7960313558578491, -0.1173718273639679, 1.0083799362182617, 0.7261773347854614, 0.49657005071640015, -0.17354224622249603, 1.4723039865493774, 0.836975634098053, -0.4859219193458557, 1.1093010902404785, -1.291106939315796, -0.7985872030258179, -0.32772693037986755, -0.8713407516479492, -0.3908122479915619, -0.5132198929786682, -1.6307156085968018, 1.169847011566162, -0.9533021450042725, 0.11251795291900635, 0.9724200963973999, 1.5931576490402222, -0.3007330000400543, -2.35752010345459, -0.1097341924905777, -0.32749783992767334, 0.29435834288597107, -1.8220425844192505, -0.3398083448410034, -0.8533872961997986, -0.39460527896881104, 1.8207229375839233, 0.23681539297103882, -1.3239121437072754, -0.2591119706630707, 0.4142996072769165, 0.6491175889968872, 1.388814926147461, -0.1340433657169342, -0.47953659296035767, -0.10266607254743576, -1.6825876235961914, 0.5715752840042114, 1.7712154388427734, -0.7189920544624329, -1.71694815158844, 0.03588539734482765, 0.7270645499229431, 0.20403265953063965, 1.2538073062896729, -1.9675995111465454, 0.0660841092467308, -0.33508405089378357, 0.4749264717102051, 0.676033079624176, -1.3791128396987915, -0.6080213785171509, -2.4781718254089355, -0.8091697096824646, -0.11483137309551239, 0.43415600061416626, 0.6102554798126221, 0.4582497477531433, -1.4169327020645142, 0.9475955963134766, 0.01559730526059866, 0.45732182264328003, 0.8253118991851807, 1.0443917512893677, -0.2950952351093292, -0.585866391658783, 0.0222773477435112, -0.04950221627950668, 0.0006951455725356936, 0.9098180532455444, 0.8736189603805542, -0.898183286190033, -1.1797317266464233, 0.06726165115833282, -0.3074602782726288, 2.6517043113708496, 0.771216630935669, -0.7167306542396545, -0.6700494289398193, 0.0769738033413887, 1.2023441791534424, 1.0009915828704834, -0.22293397784233093, 0.39192068576812744, -0.13532285392284393, 2.275132656097412, -1.3689541816711426, -0.575503945350647, -0.6219797730445862, 0.1915825456380844, 0.18503110110759735, -0.3481111228466034, -1.6446634531021118, 0.34218916296958923, 0.41277992725372314, -0.8496150374412537, 1.8711909055709839], [-0.06506778299808502, 0.09665825963020325, 0.5238931179046631, 0.6458365321159363, -0.7002291679382324, -0.033446263521909714, 1.3971881866455078, 0.08784932643175125, 0.8681405782699585, -2.0217978954315186, -0.6452300548553467, 0.17648063600063324, -1.622328281402588, -2.03985595703125, 0.011749673634767532, -1.215700626373291, -0.5070856809616089, 1.6215693950653076, 2.101714849472046, -0.5662489533424377, 0.5210190415382385, -0.07685203105211258, 0.1338614821434021, -1.4090197086334229, 0.5839316844940186, -0.12905430793762207, 0.23847752809524536, 0.3344990313053131, 1.2739582061767578, 0.9705542922019958, 0.42412182688713074, 0.6991132497787476, -0.5130191445350647, -0.5546523332595825, -0.360918790102005, -0.8706719875335693, -0.6834264993667603, -0.3171994090080261, 0.14186301827430725, 0.6285799741744995, -1.0525604486465454, 0.6129136085510254, 1.0412209033966064, 0.17011483013629913, 0.2777058482170105, -1.0348700284957886, -0.7135995626449585, -0.6207785606384277, 0.2987740635871887, 1.274505853652954, -0.24058257043361664, 2.2064905166625977, 0.3970813453197479, 0.45823073387145996, -1.7237516641616821, 2.4073052406311035, -0.5276651382446289, 1.5611270666122437, 0.06804115325212479, -0.4421416223049164, 1.2172194719314575, -0.01966201514005661, -0.41414448618888855, -0.5140446424484253, 0.17218255996704102, -0.025562966242432594, -0.18891137838363647, -0.7088424563407898, 2.3019421100616455, 2.2632219791412354, -0.14204978942871094, -0.568994402885437, 0.23860721290111542, -1.492569088935852, -0.07185906916856766, -0.22352734208106995, 0.21080145239830017, -1.4165825843811035, -0.8866570591926575, -0.01935999095439911, -0.20500096678733826, 0.2157820761203766, -0.27238744497299194, 0.430942177772522, -0.5034297704696655, -0.6673603653907776, 0.8845893740653992, 0.9804937243461609, 0.6542391180992126, -1.5381462574005127, 0.6680868268013, 1.9844536781311035, -2.0046210289001465, 0.8401106595993042, 0.175662562251091, 1.249603271484375, 1.1276310682296753, -0.9903276562690735, -1.2099697589874268, 0.9963656663894653, 0.1397029608488083, -0.207005575299263, -1.6976203918457031, 0.45527637004852295, 0.9249913692474365, 0.9793135523796082, -1.0400683879852295, -1.0589919090270996, 0.25949984788894653, -0.5305024981498718, 0.9165748357772827, 0.22999243438243866, 1.1091057062149048, 0.456769198179245, 0.9211706519126892, -1.0429874658584595, 0.5105994939804077, 1.6658021211624146, -1.1841627359390259, 2.4148783683776855, -0.009183885529637337, 0.6926301717758179, 0.1769464910030365, 0.4402436316013336, 2.349351406097412, 1.0194584131240845, 0.33226069808006287, 0.6735515594482422, 0.3399713635444641, -0.12826938927173615, -0.14884135127067566, -1.4551136493682861, -1.3632930517196655, -1.311823844909668, -0.8597401976585388, -1.697493314743042, -1.0745713710784912, 0.8352169990539551, 0.7851101756095886, 1.2104054689407349, -1.8439502716064453, -0.9187951683998108, 1.3012746572494507, 0.8890119791030884, -0.8686771392822266, 1.6542848348617554, 2.704909324645996, 1.0618520975112915, 0.04935426265001297, 0.4490377902984619, -0.3311273455619812, -0.7922570705413818, 1.0962796211242676, 0.6412456035614014, -0.3377425968647003, -0.9327114820480347, -0.9003462791442871, 0.6290535926818848, 0.31111639738082886, -1.7093461751937866, -0.4864913523197174, -0.017995242029428482, -2.807279109954834, -1.4062978029251099, -0.029076555743813515, -0.15266026556491852, -0.7226518392562866, -0.8222090005874634, -2.2470040321350098, -0.7616966962814331, 0.9792316555976868, -1.0428529977798462, -1.9879568815231323, 0.10084959864616394, -1.1034574508666992, 1.0915446281433105, 0.12238524109125137, 2.062751054763794, -0.03556160628795624, -0.022150864824652672, 0.8200533390045166, -0.8900032639503479, -0.05254792422056198, 1.2047747373580933, 0.1375080645084381, -0.7177087664604187, -0.6305266618728638, -2.2867484092712402, -1.1478803157806396, 0.29896190762519836, -0.8418041467666626, -1.464435338973999, 0.08900482207536697, 1.6486685276031494, 0.36521193385124207, -0.13472767174243927, -0.3330746293067932, -2.134932518005371, -0.5512180328369141, 0.07046732306480408, 1.179611086845398, -1.1777924299240112, -0.9878031015396118, -1.371255874633789, 0.33337512612342834, 2.262444019317627, -0.3914790749549866, 0.32966136932373047, -0.18661975860595703, -0.4399392306804657, -0.8992071151733398, 0.4837745726108551, 0.94573974609375, -0.4578796923160553, 0.6245790123939514, 0.6716915369033813, 0.3081424832344055, 0.9327478408813477, 1.0461115837097168, -0.29455745220184326, 1.3186086416244507, -0.5267745852470398, -2.2364563941955566, -0.7728023529052734, 0.4706423282623291, 0.36002692580223083, -0.1688746213912964, 0.5938636064529419, -0.18587616086006165, -1.9336888790130615, -0.1613123118877411, 0.16032159328460693, 0.7511340975761414, -0.3617250919342041, 0.1635512262582779, 0.3013092279434204, 0.3442428410053253, 0.13694623112678528, -0.0512145459651947, -0.6478070616722107, -1.863088607788086, -0.6994551420211792, -1.0947461128234863, -0.40312787890434265, -1.1949670314788818, 0.40999504923820496, -0.3840315639972687, -0.4884359836578369, 0.1385299116373062, 2.4727280139923096, 0.0361635647714138, -0.7139905691146851, 1.126301646232605, 0.5555004477500916, 1.5020912885665894, 0.6769400238990784, 1.3597992658615112, 0.002397609408944845, 0.43497371673583984, 1.1838234663009644, 0.3676789402961731, -0.001364921685308218, -0.7876569628715515, 0.4405848979949951, 1.4161452054977417, 0.8459916114807129, 1.2524006366729736, -0.10928445309400558, 0.7440856099128723, -0.7522450089454651, 0.11419013142585754, 1.1723451614379883, -0.27233585715293884, 0.49111247062683105, -0.03123733587563038, -0.25423941016197205, -1.363970398902893, 1.493363380432129, -1.8750207424163818, -0.4271356165409088, -0.12232629209756851, -0.2820124924182892, -0.10047904402017593, -2.5698511600494385, -0.5815630555152893, -0.23878978192806244, 1.3842780590057373, 0.87879478931427, 1.379807949066162, -0.8986538052558899, 1.6261963844299316, 1.7422724962234497, 0.9890145063400269, -0.36353516578674316, 0.6219529509544373, -0.1001158356666565, 1.2867118120193481, -0.006182450335472822, 0.1723249852657318, 1.4089338779449463, 0.2865034341812134, 0.8016200065612793, -1.2844491004943848, 0.8328795433044434, 1.099757194519043, 0.03317609801888466, -1.2323120832443237, 0.0786798819899559, -0.32207515835762024, 2.6794028282165527, 1.7676129341125488, 0.3844039738178253, -1.964499592781067, -0.9005035161972046, 1.0241388082504272, 1.613990306854248, 1.1676465272903442, -1.2406426668167114, 0.18365690112113953, 0.6558321118354797, 0.36805322766304016, -0.24692025780677795, -0.08513922244310379, 0.6299223303794861, 0.018064653500914574, 0.2723117172718048, -1.572642207145691, 0.3041125535964966, -0.4111994206905365, -0.1343134641647339, 0.33912554383277893, -2.291658878326416, -0.06027552857995033, 1.015043020248413, 1.859270453453064, -0.04430461674928665, -1.6655774116516113, -0.23152507841587067, -1.4651200771331787, -0.6566482186317444, -1.227664589881897, 0.801849365234375, -0.5704233646392822, 1.5420068502426147, -1.137961983680725, 1.2238856554031372, -0.19049294292926788, -1.7707983255386353, -0.7134943008422852, 0.7956029176712036, 1.111378788948059, 0.9665929675102234, -0.9711964726448059, -0.8438727259635925, 0.22735534608364105, -0.7363254427909851, -0.34933629631996155, 1.9706743955612183, 1.064997673034668, -0.8350275754928589, 0.34986037015914917, 0.3174012303352356, 1.0247366428375244, 1.056043028831482, 0.6453061699867249, -0.7019624710083008, -1.86958646774292, 0.32044461369514465, -0.29631611704826355, -1.5044128894805908, 1.3593167066574097, -1.4209171533584595, 1.5377341508865356, 1.4691730737686157, 0.11064183712005615, 0.5797150135040283, -0.46210169792175293, -1.0461926460266113, 1.1374751329421997, -0.12592251598834991, 0.6605201363563538, -1.707842230796814, 1.1622148752212524, -0.5228477120399475, 2.7314953804016113, -1.0646252632141113, 0.675649106502533, 0.46803122758865356, 0.5585681200027466, -1.2162647247314453, -1.1664025783538818, 0.6726347804069519, 0.30007317662239075, -0.18533337116241455, -0.8356781601905823, 0.6913880109786987, 0.48799726366996765, -1.2003231048583984, 0.28158432245254517, 1.8633873462677002, 0.0426056794822216, 0.7958188056945801, 1.0956605672836304, -0.23340550065040588, 0.6244475841522217, -0.598482072353363, -0.3062046766281128, -1.5655796527862549, 0.3582800328731537, 0.668391227722168, -1.3166937828063965, -1.5205382108688354, 0.07632189244031906, 2.002800226211548, 0.6735537052154541, -2.0728065967559814, -0.3245696723461151, 1.141579031944275, -0.29574206471443176, 0.07473546266555786, 0.5876708626747131, 0.004273200407624245, 1.4331848621368408, -0.3228589594364166, -1.3663194179534912, 0.8979727625846863, -0.31674841046333313, 0.4805326759815216, -1.5061837434768677, -1.0789040327072144, 1.0983299016952515, 0.06330228596925735, 0.09116269648075104, -0.3641912639141083, 0.07795710861682892, 0.9260851144790649, -0.13815470039844513, -0.8019531965255737, 1.8272011280059814, 0.15350782871246338, -0.8458169102668762, 1.3944863080978394, 0.7417865991592407, 1.0678884983062744, 1.4196066856384277, -0.3166654109954834, -2.6830270290374756, -0.7292833924293518, -1.2360998392105103, 0.23070022463798523, -2.057469367980957, -0.4860149323940277, -1.247937560081482, -0.0412343330681324, -0.2268294394016266, 0.6387395858764648, 0.24481453001499176, -2.221912145614624, -1.104333758354187, -0.33656439185142517, -0.8620299696922302, 1.765395998954773, 0.33160313963890076, -1.6619024276733398, 0.0014299298636615276, -2.635005474090576, -0.13758057355880737, 0.24635784327983856, -0.9106386303901672, 0.5664092302322388, -0.06941857933998108, 0.3087369501590729, -0.20020169019699097, -1.9905914068222046, -0.9094487428665161, -1.3876785039901733, 1.11271333694458, 1.6295217275619507, -0.058085400611162186, -1.2223976850509644, -0.15401656925678253, -2.4457554817199707, -0.5867694020271301, -0.5250818133354187, 0.37998712062835693, -1.4222400188446045, -1.9366563558578491, 0.6964688897132874, -0.3802286386489868, -1.2419379949569702, -0.1545557826757431, -0.14221172034740448, -0.5781903266906738, -0.27661794424057007, -1.4585767984390259, -0.16327281296253204, -1.914314866065979, 1.0396112203598022, 0.6134242415428162, -0.6408390998840332, 0.8447952270507812, -0.09515395015478134, -0.316037654876709, -0.7011339664459229, -0.02317911759018898, -0.4712275266647339, 0.41731390357017517, 0.9063206315040588, -0.9468603134155273, -0.21520322561264038, 0.9494338631629944, -0.6143549680709839, -0.19223706424236298, 1.2149488925933838, 0.27476733922958374, -0.4390251934528351, -0.9454219937324524, -0.19189514219760895, -0.5691075921058655, -0.46156808733940125, 0.7964653372764587, -1.8331884145736694, 0.25096598267555237, -0.5547767281532288, -1.5361748933792114, -0.45681798458099365, 1.5529989004135132, 1.9297068119049072, -0.5044662356376648, 1.065103530883789, 0.8829067349433899, -1.4160710573196411, -0.3086412847042084, -0.8877921104431152, 0.09104923158884048, -0.6251962780952454, -0.9268934726715088, -0.261089563369751, -2.6551318168640137, -0.22281280159950256, 0.24974949657917023, 0.5838639140129089, 1.173426866531372, -0.38558879494667053, -1.3668558597564697, 0.619927167892456, 0.5456498861312866, -0.8807197213172913, 0.07750944793224335, 1.4113305807113647, 1.5885958671569824, 0.7161970138549805, -0.9280906915664673, 1.4687243700027466, 0.11807239800691605, 0.6162706017494202, 0.6591102480888367, 0.29897546768188477, 1.6897093057632446, 1.2572999000549316, -1.0324161052703857, 1.6724631786346436, -0.12057633697986603, -0.6406696438789368, 0.2620069682598114, -1.426477313041687, 1.4787471294403076, 0.188801571726799, -1.0856969356536865, 0.5248519778251648, 0.3159583508968353, 0.4353654384613037, 1.1631711721420288, -1.130072832107544, -0.9249520301818848, 0.6743654608726501, 0.9151641726493835, 1.3498345613479614, 1.2142870426177979, 0.5130845308303833, 0.7497208118438721, 0.9152476191520691, -0.11162212491035461, 0.96254563331604, -0.22558440268039703, 0.17993976175785065, 0.3194930851459503, -0.06110600382089615, 0.9213396906852722, 0.6915507316589355, 0.5255674123764038, 1.2671273946762085, -0.05121321976184845, 0.48839476704597473, 1.3406293392181396, 0.11508601158857346, 1.0021605491638184, 1.4403891563415527, 0.8667429685592651, 1.4168108701705933, -2.646501064300537, -1.3816559314727783, -1.8889752626419067, 0.07419167459011078, 0.038770418614149094, -0.5096644759178162, -1.0862877368927002, -0.2745698094367981, 0.8572927713394165, 0.09028518944978714, -2.2078776359558105, -2.0394880771636963, 1.438607096672058, -0.29585373401641846, -0.07479698210954666, -0.12995338439941406, -0.47668278217315674, -0.8097384572029114, 0.8560762405395508, 1.0608110427856445, -0.33305782079696655, -1.4220508337020874, 0.3112027049064636, 0.14348217844963074, 1.349779486656189, 1.3773770332336426, 0.049233853816986084, 0.6864352822303772, 0.09392331540584564, 0.750582218170166, 0.030673542991280556, -0.07700686156749725, 0.3403882086277008, 2.4844603538513184, 0.5209746956825256, 1.6202095746994019, -0.8008029460906982, -0.2367164045572281, 0.8309215903282166, -0.6970711350440979, -1.1353987455368042, -0.49713945388793945, 1.4857959747314453, -0.8161739110946655, 0.5189600586891174, 0.6646441221237183, 0.07408928126096725, -1.6360459327697754, -0.07338247448205948, -0.593565821647644, 1.6983349323272705, -0.3961564302444458, 0.6109341382980347, -2.0024242401123047, 1.1601520776748657, -0.8098835349082947, 1.7754926681518555, 0.8030348420143127, -0.1907842457294464, -0.10001427680253983, -1.1742700338363647, -1.2570751905441284, -2.07302188873291, 0.4948602020740509, 0.7948870062828064, 0.0705246701836586, -0.1383659839630127, -1.669220209121704, -2.0076584815979004, 2.174363136291504, -1.3506180047988892, 1.320831060409546, -0.6067173480987549, 0.05190162733197212, -0.9157127141952515, 1.3050216436386108, -0.9841781258583069, -0.8595250248908997, 1.1513781547546387, 1.271713137626648, -0.8473796248435974, 1.1281384229660034, 1.0885852575302124, 0.8408591747283936, -1.6565340757369995, 0.21572311222553253, 0.11158668994903564, 0.2856360375881195, 1.4952055215835571, -1.1138547658920288, -0.19653019309043884, -1.2437697649002075, 0.49286574125289917, -0.18176180124282837, -1.4530858993530273, -1.9898889064788818, -1.3607511520385742, 0.07382739335298538, 0.22917701303958893, -0.7243674397468567, -0.771568775177002, -0.007365958299487829, -1.1336404085159302, -0.4281145930290222, -0.7364579439163208, -0.5003610849380493, 0.07733689993619919, -0.4956595301628113, -1.5946341753005981, -0.17876091599464417, -0.24791203439235687, 0.07599640637636185, 0.9000927805900574, -0.04729941114783287, 0.47082582116127014, -0.5126408934593201, 0.36554795503616333, -0.390237420797348, -0.14402273297309875, -1.664283037185669, 1.7605186700820923, -0.2908609211444855, -0.7895361185073853, 0.24142204225063324, 0.05542069301009178, 0.7389962077140808, -0.19565743207931519, 0.05921212583780289, -0.21892373263835907, 2.048769950866699, -1.25685453414917, 0.48933809995651245, 0.019636860117316246, -0.15609648823738098, 0.7401025891304016, -1.0963959693908691, 0.17206443846225739, 0.8510164618492126, -0.3590797483921051, 1.1237725019454956, -0.8334943652153015, -1.2269530296325684, 1.2488633394241333, 0.5893138647079468, 0.9525924324989319, -0.9832844138145447, -0.20767897367477417, 0.2980571389198303, -0.1421838253736496, -0.13769401609897614, 0.140369713306427, -2.2712900638580322, -1.0450561046600342, 0.7748550176620483, 0.3930325508117676, 1.6682419776916504, -1.3287639617919922, 0.5678503513336182, -0.17424920201301575, -0.31529858708381653, 2.154541015625, -1.131457805633545, -0.30045393109321594, 1.1930599212646484, 1.8436585664749146, 0.6304458975791931, -0.13444072008132935, -1.3174837827682495, 0.5294831395149231, 1.1797517538070679, 2.506159543991089, -0.3303380608558655, 1.0062109231948853, -0.5815590620040894, 1.1711645126342773, 0.9102969169616699, -0.2779470980167389, 0.2001345157623291, -0.7828632593154907, 1.6580530405044556, 0.2681458294391632, 0.7939019203186035, 0.40889161825180054, -1.027255654335022, 1.06251859664917, 0.8518609404563904, 0.9436297416687012, 0.45058879256248474, 0.20764966309070587, 0.3450184762477875, 0.3373878300189972, -0.2502336800098419, -0.9038903713226318, 0.3488085865974426, 1.7026071548461914, 0.06943682581186295, -0.33105790615081787, 0.9128264784812927, -0.8937373757362366, 0.17236357927322388, -0.8463719487190247, -0.2425495833158493, 0.22568431496620178, 1.363816738128662, -0.49204155802726746, 1.8204386234283447, -0.022899525240063667, 0.5190712213516235, 1.3541170358657837, 0.21781519055366516, 0.4093632698059082, 0.46000492572784424, -0.2841084897518158, 0.8979423642158508, 0.5728746056556702, -0.6887845396995544, 1.075904369354248, -0.9062358736991882, -0.9197418689727783, -1.3237922191619873, 0.8396545648574829, 1.2375280857086182, -0.3287624716758728, -1.2739895582199097, 0.4961044192314148, -1.3999532461166382, 0.9433308243751526, 1.344465970993042, -0.8364172577857971, 1.0587944984436035, 0.038049887865781784, -0.6962939500808716, -0.11297592520713806, 0.9564228057861328, 0.5939569473266602, -0.3609224259853363, -0.33239585161209106, 0.9900901317596436, 1.2259548902511597, 1.327300786972046, -0.5308422446250916, 0.33111122250556946, -0.0034410732332617044, 1.1437668800354004, 1.4182168245315552, -0.8769737482070923, -1.25925874710083, 0.729038655757904, -1.449628472328186, 0.9574452638626099, 0.10243934392929077, -1.4499881267547607, -1.4696197509765625, 0.6367148756980896, -1.572535514831543, 0.9129093885421753, -0.18094812333583832, 0.8693627715110779, 0.6040239930152893, 0.7799379825592041, -0.5955169200897217, -0.35887837409973145, -0.8302825689315796, -0.08852724730968475, 0.7405796647071838, -0.1828596293926239, -1.5247337818145752, -0.9630892276763916, 0.47121819853782654, 0.8047494888305664, 1.5614278316497803, -0.6491675972938538, -1.7810657024383545, 0.717974066734314, -0.44777771830558777, -1.4000256061553955, -0.27412089705467224, -0.34902167320251465, -0.559207558631897, -1.4989794492721558, -0.24136213958263397, -1.4195821285247803, 1.1802793741226196, -0.9848949313163757, 0.04085038602352142, 0.3538026511669159, -0.5407769083976746, 0.6335286498069763, -0.011898115277290344, -0.6983683109283447, -1.4945536851882935, -2.1476831436157227, -0.4434271454811096, -0.3436175286769867, -1.6593137979507446, 0.9480231404304504, 0.032473742961883545, -0.6400036811828613, 0.6571608185768127, -0.6966642737388611, 0.28365838527679443, -0.5048655271530151, -1.1918339729309082, -0.18558980524539948, -0.2021406888961792, 2.1384167671203613, -0.5654873847961426, -0.786397397518158, -1.9587233066558838, 1.368026614189148, -1.4970148801803589, 2.257610559463501, -0.43720266222953796, 0.10051938146352768, -0.26906898617744446, 1.8063350915908813, 1.115773320198059, 1.0585600137710571, -1.5478192567825317, -2.207318067550659, -2.1428213119506836, 1.4813822507858276, -0.5009480118751526, 0.9902865290641785, 0.014803007245063782, 3.265653610229492, -0.5742372870445251, 1.2408876419067383, -1.3456178903579712, 0.05462641641497612, -0.1853557825088501, 0.4099878966808319, -0.46977895498275757, -0.937538206577301, -0.5770170092582703, -1.1632288694381714, 0.07840866595506668, 0.34076422452926636, 0.33875781297683716, 0.4563967287540436, 0.7557945847511292, -0.7117438316345215, -0.23237812519073486, 0.851777195930481, 0.2467876672744751, 0.9735891222953796, -1.5221797227859497, 0.49538102746009827, 0.8909046053886414, 0.33920979499816895, 0.6804440021514893, 1.3992571830749512, 0.46961769461631775, -0.18041282892227173, 0.4416005313396454, -0.3158118724822998, -0.36434900760650635, -0.01519447099417448, -2.7645294666290283, -0.4642198383808136, 0.07807054370641708, -0.2821357548236847, -1.0234487056732178, 1.0817629098892212, 0.33591729402542114, 0.44253066182136536, -0.4977003037929535, -0.9296151995658875, -0.6139028072357178, 0.6866415739059448, 1.9343727827072144, -1.6455786228179932, -0.9682385325431824, -0.7384750843048096, -1.3185254335403442, -0.7218866944313049, 0.08981561660766602, 0.6889451146125793, -1.177746057510376, 0.8373778462409973, 0.48781561851501465, -1.7272958755493164, -1.9325112104415894, 0.6982401013374329, 1.4073599576950073, 0.6508880853652954, 0.39488059282302856, 0.22401951253414154, 0.07409419119358063, 0.8939189910888672, 0.7162374258041382, -1.1273763179779053, 0.4221622347831726, 0.2791571617126465, -0.08809521794319153, 0.29772090911865234, -0.09863197058439255, 0.010133268311619759, -0.1987597495317459, 1.058656930923462, 1.3918814659118652, -0.007913759909570217, -0.3596506118774414, -1.6829612255096436, -0.5354544520378113, 0.5824429988861084, 0.8455708622932434, 1.8255738019943237, -0.91947340965271, 1.0330311059951782, 0.922928512096405, -0.7670684456825256, 1.2312575578689575, -0.6561574339866638, 0.26121965050697327, -0.7532406449317932, 0.6534480452537537, -0.26416003704071045, -0.8803832530975342, -0.4441995322704315, -0.01762736402451992, -1.5001142024993896, 0.45147979259490967, -0.36987197399139404], [-0.7981821894645691, 0.07322744280099869, -0.4504951238632202, -0.3324030339717865, -0.6013773083686829, 0.8004042506217957, -0.3637882471084595, 0.416897177696228, 0.016752513125538826, 1.9331607818603516, 0.5762201547622681, -1.2022705078125, 1.0079761743545532, 0.6971665620803833, 0.547205924987793, -0.5561558604240417, 1.3090561628341675, -2.231667995452881, -1.3584275245666504, 0.32925158739089966, -0.8015483021736145, 1.4554718732833862, -0.5965381860733032, 1.2235174179077148, -2.258819818496704, 1.516666293144226, -0.7218443751335144, -0.1791764497756958, -0.28725215792655945, 0.3862864673137665, 0.9409869909286499, -0.8830304741859436, -0.7514461874961853, 0.6224327087402344, 0.9002411365509033, -1.4132767915725708, -0.6694353818893433, 0.33032917976379395, 0.42935237288475037, -3.1929678916931152, 0.926689088344574, -0.3791108727455139, 0.7565717697143555, -1.1657085418701172, -1.231619119644165, 0.8494077324867249, -0.31899383664131165, -1.0130045413970947, -1.3315919637680054, 1.5935581922531128, -0.26053813099861145, -0.8092584013938904, -1.1843675374984741, -3.67569637298584, -1.413880467414856, -0.32312095165252686, -0.7765471935272217, -1.4356945753097534, -0.19520647823810577, -0.6444183588027954, -0.552757203578949, -1.85675847530365, -1.2724932432174683, 0.7263041734695435, -0.13242094218730927, 0.6927618980407715, 0.7481884956359863, 1.7447400093078613, 1.16793692111969, 2.1612725257873535, -1.8777427673339844, 0.5928064584732056, -0.23699073493480682, 1.1779910326004028, -0.07550785690546036, 0.012157170102000237, -0.8261753916740417, -1.7006269693374634, -1.1442166566848755, 0.7299162745475769, 0.42959341406822205, 1.2361359596252441, 0.051632530987262726, 0.44431623816490173, 1.3195258378982544, -1.015445351600647, 1.3720568418502808, 1.3875622749328613, 0.07389384508132935, 1.4104325771331787, -0.8521772027015686, -0.03674965724349022, -1.011390209197998, -0.7457892894744873, 0.19686970114707947, 0.7958568930625916, -1.1426998376846313, 0.031374383717775345, -1.2171541452407837, -1.2328943014144897, 1.5499165058135986, 1.827273964881897, -1.482015609741211, -0.38478556275367737, -0.37123969197273254, -1.024247646331787, 2.6484432220458984, 0.6202121376991272, -1.0419418811798096, -0.8541882038116455, 1.396504282951355, -0.8957628607749939, -0.4539722800254822, 0.32169967889785767, 0.1704232096672058, 0.5735604763031006, -0.45092639327049255, 0.10426276922225952, 0.35293444991111755, -2.4056527614593506, 0.30290406942367554, 1.2767541408538818, -0.3941887319087982, 1.2484986782073975, 0.16421975195407867, 0.22616258263587952, 0.42683106660842896, 0.17609596252441406, -1.3758482933044434, -0.7636950016021729, -0.40365415811538696, 1.3470407724380493, -1.0109025239944458, 1.0392637252807617, 1.539313554763794, 0.7268335223197937, -0.6525578498840332, 0.7151426672935486, -0.08016008138656616, 0.12975114583969116, 0.44373515248298645, -0.7704852223396301, 1.5602812767028809, 1.7701342105865479, 0.17456965148448944, 1.1694860458374023, 0.531427800655365, -0.3638518452644348, -0.8950810432434082, -1.419861078262329, -0.8222982287406921, -2.2003016471862793, 1.4525415897369385, 0.3733297884464264, 1.8278310298919678, 1.0829689502716064, 1.514573335647583, -0.15005534887313843, -1.4601037502288818, -0.3733556568622589, 2.188223361968994, 1.3322818279266357, 2.1716012954711914, -1.6226774454116821, -1.1871360540390015, 1.8835978507995605, -0.5678071975708008, 0.5638352632522583, 0.9915809035301208, -0.42141997814178467, -1.6112096309661865, 0.8859735131263733, -0.3053877353668213, -0.5456290245056152, 1.0572248697280884, -1.1981512308120728, 0.34356889128685, -0.10857818275690079, 0.6795724630355835, -0.16681799292564392, -1.5516525506973267, 0.5348512530326843, 0.719490647315979, -0.03615158423781395, 0.41600698232650757, -0.9204358458518982, 0.6605333089828491, 0.3196466565132141, 0.3693119287490845, -0.7051531672477722, -0.0023697561118751764, -0.6365273594856262, 0.8411976099014282, -0.30310478806495667, -0.03898118436336517, 0.2197917401790619, -0.7029078006744385, 0.3187817931175232, -2.140279531478882, 0.614785373210907, 0.7143574953079224, -1.3382421731948853, 0.6092547178268433, 2.542173385620117, 0.43474826216697693, -1.1814377307891846, 1.0113940238952637, -0.2389211505651474, -0.3313772976398468, -1.0846928358078003, 0.16462597250938416, 2.3386030197143555, -0.9691969156265259, 1.1252833604812622, 1.6896573305130005, 1.0856770277023315, -0.9008192420005798, 0.8467243313789368, 0.6300652027130127, -0.24137882888317108, -0.49014022946357727, 0.17606033384799957, -1.3990428447723389, -0.6565088033676147, -0.18659693002700806, -0.8652066588401794, -1.967573881149292, 1.251221776008606, 0.2057337462902069, -0.7064226865768433, 1.1703338623046875, 0.662098228931427, 0.19859616458415985, -0.3238513469696045, -0.3776512145996094, 1.3875694274902344, -0.4805126190185547, 0.7041898965835571, -0.6998748779296875, 0.416029155254364, 0.6702790260314941, -0.08178962022066116, -0.06769061088562012, 1.250428557395935, -1.0868662595748901, 1.0609265565872192, -0.06182314082980156, 0.9635956287384033, 0.7076339721679688, 0.1965150237083435, -0.39074885845184326, -0.27205610275268555, 0.21708528697490692, -0.6115922927856445, 1.5527065992355347, 2.1378393173217773, 1.058456301689148, -0.35358428955078125, 0.9988470077514648, 0.19615623354911804, -0.7536872625350952, 0.6231635808944702, -0.6919682025909424, -0.8736950755119324, -0.9975873231887817, -1.6203796863555908, 0.12377016246318817, 0.3940623104572296, -0.5546473860740662, 0.11539584398269653, -0.4842688739299774, -1.2705705165863037, -0.6738092303276062, 1.2392990589141846, -0.004150948021560907, 0.6775169372558594, 0.8850979804992676, -0.4370942711830139, 1.0279160737991333, -1.0819439888000488, 0.9683173894882202, -0.3583507537841797, 1.0369702577590942, -1.642535924911499, -0.015603854320943356, -0.9828274846076965, -0.09067308902740479, -0.07786457985639572, 0.3386435806751251, 0.10634569823741913, -0.8583365082740784, 0.08477555960416794, 1.7242705821990967, 0.8831087350845337, 0.12146031856536865, 0.6460111141204834, -0.1229085847735405, 0.5612009763717651, -1.5790042877197266, 1.4336321353912354, 0.46314379572868347, 1.6297392845153809, 0.3376018702983856, -0.6112105250358582, 0.11887131631374359, -0.6319552659988403, -0.07440268993377686, 0.8025768995285034, 1.8999024629592896, -1.250123381614685, 1.7943470478057861, -0.7659595012664795, -0.7869051694869995, 1.0078709125518799, -0.5438122153282166, 0.46652951836586, 2.7517282962799072, 0.634151041507721, -0.36408156156539917, -0.5855944156646729, -0.6541609168052673, -0.9008133411407471, -0.269939661026001, -0.5552317500114441, -0.12662208080291748, -0.7225102186203003, -1.1951557397842407, -0.05252687633037567, -0.09173239767551422, -0.9425250887870789, 0.719312310218811, -0.5081251859664917, 1.8203885555267334, -0.5772931575775146, -0.29830417037010193, -0.6485325694084167, -0.2541804015636444, -0.588631808757782, -0.26248234510421753, 0.8224684596061707, -0.4669976234436035, 2.5886268615722656, 0.7139532566070557, -1.1723158359527588, 0.6738888025283813, 0.29649239778518677, -1.1000295877456665, -1.529353380203247, 0.2635403871536255, -0.5802934169769287, 0.5340374708175659, -0.5431542992591858, -0.34273505210876465, 0.17506688833236694, 0.389849454164505, -0.09563989192247391, 0.20689156651496887, -1.416509747505188, 2.0820233821868896, -1.556225299835205, 0.05306246131658554, -0.40432190895080566, -0.5772719383239746, 1.872388482093811, -1.7086050510406494, -1.7859396934509277, -0.5774016380310059, -0.9703195095062256, 0.8347251415252686, 1.5902212858200073, -1.332415223121643, 0.009501558728516102, -1.1229543685913086, 2.388144016265869, 0.15831032395362854, 0.3492737114429474, 0.08873078972101212, -1.3383631706237793, 0.5449498295783997, 0.23629195988178253, -0.9015985131263733, -2.224623203277588, -0.5232104659080505, -0.6882680058479309, 1.9848066568374634, -0.07157013565301895, 1.2192659378051758, -1.1370410919189453, 0.1931002289056778, 0.7837573289871216, 0.07235915958881378, 0.7787714600563049, 0.7740333080291748, 0.7373049855232239, -0.6410958170890808, -1.611567735671997, -0.3642888069152832, -0.514801025390625, 0.8232874870300293, -0.4248185455799103, 0.42504292726516724, -2.6625730991363525, 0.8698319792747498, -0.418133944272995, 0.5716239809989929, 0.6115754842758179, 0.670007586479187, 0.35968661308288574, -2.0683822631835938, -0.10099343955516815, 0.10282531380653381, -0.3270963728427887, -0.3755514919757843, -0.2443874180316925, 0.40159520506858826, 0.8236033916473389, -0.03361842781305313, -0.28449609875679016, -3.324326992034912, 1.4898364543914795, 1.3262040615081787, -1.1867843866348267, 0.3795816898345947, -2.086526393890381, -0.07163605093955994, -0.5373755097389221, -0.906610906124115, -0.38992878794670105, -0.561090886592865, -0.5929192900657654, 0.21938377618789673, 0.09459196031093597, 1.324012041091919, 0.8523851633071899, -0.8218984007835388, -1.2927778959274292, 2.14137864112854, 0.8818760514259338, -0.9493914246559143, -0.6485334634780884, 0.9148112535476685, -0.16520221531391144, 1.0090292692184448, 0.30506670475006104, -0.687321126461029, 0.15629185736179352, -1.2051568031311035, -0.8496896624565125, 1.086270809173584, -0.15338566899299622, 0.6857398748397827, -0.1760377585887909, -0.4073651432991028, 1.752274990081787, 1.78399658203125, -0.20437124371528625, 0.7145316004753113, -0.22677072882652283, -0.6958025097846985, 0.011364487931132317, 1.2603707313537598, -0.9651910662651062, -0.2657211422920227, -0.44543367624282837, 0.36306893825531006, 0.43435579538345337, 0.805733859539032, 1.2038118839263916, -0.21536444127559662, -0.8443306088447571, -0.5177886486053467, 0.8371450304985046, -0.5637539625167847, -2.313328981399536, 1.8542038202285767, -0.905012845993042, 1.0839319229125977, 0.06417670845985413, 3.03680157661438, 0.02311556600034237, -0.6403629183769226, 0.8712521195411682, -1.1692113876342773, -0.7608823180198669, -0.8968740105628967, 1.5277680158615112, -0.9752901792526245, -0.9293095469474792, -0.8885441422462463, 1.2692207098007202, 0.826130211353302, -0.3942355811595917, 0.3159938454627991, 0.1651073694229126, -1.5687193870544434, -2.213726282119751, -0.3678305745124817, 0.5969133973121643, 0.8262578248977661, -0.02729971706867218, -0.16250045597553253, -1.7243512868881226, -0.4078839421272278, 0.040578603744506836, 0.5225396156311035, -0.8773221969604492, 0.9067205190658569, -0.6290550231933594, 1.4372835159301758, -0.12956474721431732, -0.3995489180088043, -0.2893381714820862, 1.799397587776184, -1.400273323059082, -1.3608863353729248, -0.23139126598834991, -0.786022424697876, 0.046755291521549225, -1.9108192920684814, 1.5695300102233887, 1.3938019275665283, -0.32895374298095703, 1.0149320363998413, 1.2760834693908691, -2.8657965660095215, 0.10697653889656067, 0.6002458333969116, 1.208508014678955, 0.9219329953193665, -1.2325479984283447, 0.5457689762115479, -0.9205337762832642, 1.9340341091156006, 1.6758906841278076, 0.2690623998641968, -0.7330529689788818, 0.5319867730140686, 0.1452905535697937, -1.224868655204773, 0.12886348366737366, -2.6262147426605225, 0.2953645884990692, -1.4659255743026733, 0.4339514672756195, -1.4407354593276978, 1.0475356578826904, -0.5466823577880859, 1.1074028015136719, -0.014965125359594822, -0.1837732344865799, 0.4248346984386444, 0.17208217084407806, 1.4919676780700684, 0.022769305855035782, 0.20066377520561218, 0.8407758474349976, 0.04897454380989075, -0.40928730368614197, 2.0079843997955322, 1.032330870628357, -1.3859542608261108, -0.8742113709449768, -1.6201437711715698, -0.031575679779052734, 0.05276603251695633, -0.05183287337422371, -0.7715065479278564, -1.3135379552841187, 0.4704097807407379, 0.16500310599803925, 0.37888994812965393, -1.616998553276062, -0.4818131923675537, 1.5543090105056763, -0.03908317908644676, 0.8618743419647217, -0.09408733993768692, 0.043206218630075455, 0.9104230999946594, 1.632328987121582, 0.16992168128490448, 2.9174208641052246, -0.1760408580303192, -0.3122289180755615, 1.3254759311676025, -0.5911279320716858, -0.6283186674118042, 0.2611839771270752, -0.4967556595802307, -1.0739071369171143, 0.6955084204673767, -0.1394503116607666, 0.7339978814125061, -0.12769393622875214, 0.19866105914115906, -1.0114060640335083, 0.5203701257705688, -0.2790001332759857, -0.32705235481262207, -0.23586460947990417, 1.8384130001068115, 0.1809154450893402, -0.0413585901260376, -0.35958409309387207, -1.0442394018173218, -0.6268554925918579, -2.0908892154693604, -0.29071083664894104, 0.4742659032344818, 0.8112518787384033, -0.715133786201477, -0.5431134700775146, 0.0829029530286789, 0.7401214838027954, -1.1227998733520508, -0.7963528633117676, 1.7464271783828735, 0.2193576842546463, 0.7997284531593323, -1.1735457181930542, -0.648968517780304, -1.8210285902023315, -1.6092820167541504, -1.0893925428390503, 1.5942366123199463, -0.11758210510015488, -1.4065607786178589, 0.9912852644920349, -1.2712100744247437, -1.1366689205169678, 0.7382129430770874, -1.444602370262146, -1.2456549406051636, -0.9293086528778076, 0.4255744218826294, 0.05215046554803848, 0.14184711873531342, 0.645072340965271, -0.33692631125450134, 0.7591145038604736, -0.5676330924034119, -0.07399848848581314, -0.1570897400379181, 2.20646333694458, 0.5342750549316406, -0.09431931376457214, -0.38957569003105164, 1.4307291507720947, -0.6425041556358337, -1.0918456315994263, -0.056952159851789474, -1.7563425302505493, -0.13186822831630707, -0.027446439489722252, 2.1388001441955566, 1.499221682548523, -0.8324189782142639, 1.304748773574829, -1.013319730758667, 1.6913485527038574, -1.2550190687179565, 1.7251050472259521, -0.11199173331260681, 0.27067771553993225, -0.1113780215382576, 0.4171502888202667, -0.05182098224759102, 0.29511135816574097, 1.3909882307052612, -1.0945532321929932, 0.2605951726436615, 0.1586056649684906, 1.9222735166549683, -0.39119404554367065, -0.6488214135169983, 0.1253957450389862, 0.3445458710193634, -1.2817773818969727, 0.15821439027786255, -0.9436759352684021, 0.04920555651187897, 0.7141020894050598, 0.03550915792584419, -0.8088251948356628, -0.8808345794677734, 0.8399982452392578, -0.33725085854530334, 2.2948999404907227, 0.34605148434638977, 1.4825555086135864, 1.8413335084915161, -0.902899980545044, 0.3983726501464844, 1.3197003602981567, -0.5039026141166687, -0.17926150560379028, 1.3963079452514648, -0.6137582063674927, 2.78776216506958, 0.9480112791061401, -0.3182697296142578, -1.186342716217041, 0.837062656879425, 0.5444878339767456, -1.5532739162445068, -1.243866205215454, -1.1160677671432495, 1.311739206314087, -0.023273633792996407, -0.410575270652771, -1.3617693185806274, 1.499841332435608, 1.3343042135238647, -1.7725143432617188, -0.4479752480983734, 0.6906052231788635, -0.09003371745347977, -0.19990994036197662, 2.1379995346069336, -0.05982178822159767, 0.051643021404743195, -0.4412049353122711, 0.4620952010154724, -0.11704804003238678, -0.47999972105026245, -0.03973779082298279, 0.04812077060341835, 0.7877895832061768, -1.2406115531921387, -1.4020622968673706, 1.0614683628082275, -1.0400997400283813, -0.37660518288612366, -0.7306263446807861, 0.5160281658172607, 1.3425410985946655, 0.5538555979728699, 0.6034526824951172, 0.4861160218715668, 0.3165225684642792, -0.9050015807151794, 0.2223813831806183, 0.6402414441108704, 1.1041873693466187, 0.01051202230155468, 0.6695213317871094, -0.7407591342926025, -0.36159035563468933, 0.37375596165657043, 0.801442563533783, -0.3870028257369995, -1.6159164905548096, 0.885539174079895, 0.0646623969078064, 2.0983879566192627, -0.6406728625297546, -0.7587694525718689, -0.9182473421096802, -0.1790228933095932, 0.33173736929893494, -0.4750917851924896, 0.24841181933879852, -1.7290691137313843, -0.05236082524061203, -1.6416237354278564, 1.504647135734558, -2.5596706867218018, -1.42714262008667, 0.7167704105377197, -1.2981926202774048, 1.9026743173599243, 0.4906902611255646, -0.6939637660980225, 1.8269386291503906, -0.4963536262512207, -0.17857995629310608, -0.49148231744766235, 0.7479889392852783, -0.29636773467063904, -0.1354731172323227, -0.6432431936264038, 2.091221332550049, -0.4643195867538452, 0.14784912765026093, 0.8199073672294617, 1.279945969581604, -0.2841241657733917, -1.5366672277450562, 1.200374960899353, -0.814326286315918, 0.21430562436580658, -1.090956211090088, 0.07394292205572128, 2.0015032291412354, -1.7813198566436768, -0.5307040214538574, -0.17894448339939117, 1.218186616897583, -0.7207963466644287, 0.1356751173734665, -0.015273697674274445, 0.11909215152263641, -0.12274142354726791, -1.7848156690597534, -2.184328079223633, 0.05997607111930847, 1.8512457609176636, -0.7786223292350769, -1.1661192178726196, 1.5628894567489624, 0.4328232705593109, 2.014000654220581, -1.7352290153503418, 0.44189688563346863, -0.8369996547698975, 0.010051961988210678, -0.6084306240081787, -1.897227168083191, 0.43209555745124817, 0.2474985420703888, 0.5131857991218567, 0.48119327425956726, -0.4558565020561218, 0.21868298947811127, 0.8680554628372192, 0.8552688360214233, -0.5481253266334534, -0.3594458997249603, -0.014751248061656952, 0.2874571681022644, 0.07652362436056137, 0.006714577320963144, 0.1581689715385437, -1.1782575845718384, 0.7408012747764587, 0.8149341940879822, -0.4339888393878937, -1.0846331119537354, 0.23889701068401337, -0.020769311115145683, -0.9169893264770508, -0.3207874298095703, 1.4121979475021362, 1.0119036436080933, 1.214521884918213, 0.6662288308143616, 1.0111843347549438, -0.5395118594169617, -1.016109824180603, -0.5060592889785767, 0.44269633293151855, 0.6168494820594788, 0.19476011395454407, 0.39928972721099854, 1.7900261878967285, -0.46132004261016846, 1.2683852910995483, 0.36667993664741516, 0.16184453666210175, -0.7988629937171936, 0.4428427219390869, -0.8632080554962158, -1.8335740566253662, 0.5835472345352173, 0.7800165414810181, 2.459318161010742, -1.0223102569580078, 0.6288567781448364, -1.2418997287750244, -1.0805799961090088, -0.7608864307403564, -0.4114382565021515, 0.21611246466636658, -0.34468793869018555, -0.14955861866474152, 2.6313424110412598, 0.4932093024253845, -0.6752642393112183, 0.9278588891029358, 0.5097257494926453, 1.7749865055084229, 0.6009992361068726, 0.7086447477340698, -0.02419818378984928, 0.08376947045326233, -0.2688242197036743, -1.2625014781951904, -1.0100733041763306, -0.29764634370803833, -0.784199059009552, 0.6139183044433594, -0.3033585548400879, 0.08702961355447769, 1.8386691808700562, 1.8881398439407349, -0.39005449414253235, -0.6368831992149353, -0.8384519815444946, 0.04303370788693428, -0.7771946787834167, -0.5408708453178406, -1.5096025466918945, -0.4013386368751526, 0.5233265161514282, -0.39679399132728577, -1.6957547664642334, -0.11473994702100754, 0.6487957239151001, 0.8708117604255676, 1.000573992729187, 0.047807417809963226, 0.00420222757384181, -0.9381831288337708, 0.8557884097099304, -1.6584182977676392, 0.618041455745697, -1.837278127670288, 0.3371259272098541, -0.09705499559640884, 0.4513319134712219, 1.468847632408142, -0.16378580033779144, -0.08419676870107651, 1.8139007091522217, -1.59526526927948, -0.8403817415237427, 0.23148970305919647, 0.06394047290086746, -0.028337392956018448, 0.5753054022789001, -1.2479872703552246, -1.890551209449768, 0.6407544016838074, -1.6944048404693604, 1.1775459051132202, -2.354918956756592, -0.7934191226959229, 0.21631352603435516, -0.7969652414321899, -0.37481263279914856, 0.6851329207420349, 0.8763558268547058, 1.1495095491409302, 1.6078475713729858, 1.7880655527114868, 0.08249017596244812, 0.17687052488327026, -1.5956978797912598, -0.2552754878997803, -0.6775564551353455, 1.8674386739730835, 1.9718834161758423, 0.6978954672813416, 0.22364795207977295, -0.1234622523188591, -0.7942898273468018, 0.29871582984924316, -0.551695704460144, 0.740858256816864, -1.1149617433547974, 1.066465139389038, 1.9330540895462036, 1.407515287399292, -0.834962010383606, 2.7234199047088623, -0.11241783946752548, 0.404721200466156, 1.0280826091766357, 0.753903865814209, -0.5433156490325928, -1.2534583806991577, 1.3287503719329834, 0.26272451877593994, -0.11060553789138794, 0.5718977451324463, -0.4048924148082733, 0.2107734978199005, -0.4207916557788849, -0.2095242738723755, 0.5622565746307373, 0.13911695778369904, 0.16653382778167725, 0.1539735347032547, 2.0363967418670654, -1.3229668140411377, 0.14781159162521362, -0.5425620675086975, -0.6299949884414673, 0.15007436275482178, -0.4068402349948883, 0.7755457758903503, 0.4705055058002472, 0.323970764875412, 2.391630172729492, -0.42275136709213257, -0.5602700114250183, 0.03234628587961197, -0.47623103857040405, -0.15706615149974823, -1.222361445426941, -0.7822433114051819, -0.7956809997558594, -0.19292853772640228, -0.1571376919746399, -0.3259357213973999, -0.5900692343711853, 0.6043720245361328, 0.93259596824646, -0.3372351825237274, 0.7952998280525208, -0.22207111120224, -0.39424845576286316, 1.0486419200897217, -0.38977763056755066, -1.1918926239013672, 0.34332361817359924, -0.2934871017932892, 0.48836079239845276, -0.49698004126548767, 0.058005280792713165, -1.6719409227371216, -1.6628170013427734, -0.5532554388046265, -0.1433645784854889], [-1.0907255411148071, 1.4750291109085083, -0.5110781788825989, 2.072031259536743, 0.6992842555046082, -1.0367701053619385, -0.3309133052825928, 0.17512232065200806, -0.19883045554161072, 1.4391449689865112, 1.5876086950302124, 0.7130408883094788, 1.648158073425293, -0.553787887096405, -0.22439876198768616, -0.5501937866210938, -0.7738814353942871, 0.2828836441040039, 0.9072325229644775, -0.9704510569572449, 1.565708041191101, -0.10394201427698135, -1.7717238664627075, -0.42875728011131287, 0.19748124480247498, 0.5337659120559692, 0.1613568216562271, 0.7936418056488037, -0.09858498722314835, 0.4630395472049713, 0.25276726484298706, 0.11027226597070694, 0.8703691363334656, 0.729057788848877, -0.4779343605041504, 1.225507140159607, 1.9211102724075317, -0.6281543970108032, -0.16724169254302979, -0.13562220335006714, 0.07532747089862823, -1.554542899131775, 0.4821838140487671, 0.2799009680747986, 1.5250740051269531, -0.10670171678066254, 0.6102608442306519, -0.7257431149482727, 0.593085765838623, 0.15434150397777557, 0.09848619997501373, 0.8347095251083374, -1.608379602432251, 0.2376280426979065, 0.6551975607872009, -0.00018796978110913187, 0.6307156085968018, -0.19978214800357819, 0.34321555495262146, -2.026205539703369, 0.9659305810928345, -0.4301750063896179, -0.1420949399471283, -0.9351308941841125, -0.3884093165397644, -0.3946322500705719, 0.09378698468208313, 0.4271047115325928, -1.6909611225128174, -0.584705650806427, -0.20712487399578094, 0.3442703187465668, -1.03560471534729, -0.8052911162376404, -0.5204275846481323, 0.4283467233181, 0.023554760962724686, -0.5777159333229065, 2.2230336666107178, 0.27073556184768677, -0.13607022166252136, -0.7193428874015808, -1.131529688835144, 0.7556505799293518, 0.07286576926708221, 0.3502083718776703, 0.5675170421600342, -0.27141597867012024, -0.5659281015396118, 2.660675048828125, 0.575171709060669, -0.22442947328090668, 1.3408474922180176, 0.6479601263999939, 0.7606853246688843, -0.10709148645401001, 0.2441243976354599, 2.684424877166748, -0.9327342510223389, 0.26518523693084717, 0.746688723564148, 1.8061619997024536, 0.2786334753036499, -0.7508422136306763, 1.4159307479858398, 1.3519407510757446, 1.4285099506378174, -1.3916845321655273, 0.8323335647583008, 0.0712764710187912, 0.26613616943359375, -0.978643000125885, -1.2155345678329468, 1.172805905342102, 2.2120492458343506, -0.052859388291835785, -0.19236770272254944, 1.0236469507217407, -1.08714759349823, -0.3747263550758362, -1.6956593990325928, 1.293405294418335, 0.349275141954422, 0.35232630372047424, 0.48218825459480286, -0.1880612075328827, 0.7905791401863098, -0.14944863319396973, 0.271590918302536, 0.5703514814376831, -0.1018768772482872, 0.21660195291042328, 1.0674625635147095, -0.6195642352104187, 0.14871053397655487, -0.602531373500824, 1.2347019910812378, 1.237368106842041, -1.6037906408309937, -1.3346359729766846, -0.2665577828884125, -0.6720373034477234, 0.26401957869529724, -0.14468568563461304, -0.656391978263855, 0.04750336334109306, 1.9429678916931152, 0.8528696894645691, 0.23785586655139923, -0.3391861021518707, 1.4679464101791382, 0.8564168214797974, 1.3506048917770386, -0.3186132311820984, -0.7631211280822754, 0.134458526968956, -0.16587381064891815, -0.24316126108169556, -1.1412688493728638, 1.0312732458114624, -0.6291945576667786, -1.106942057609558, 0.7169802188873291, -0.18389847874641418, -0.7202909588813782, 0.009610111825168133, 0.4591121971607208, -0.25161588191986084, 0.23294299840927124, -2.1168434619903564, -1.0093262195587158, 1.9296764135360718, 0.490062952041626, 0.9678279757499695, -1.4897346496582031, 0.2154339998960495, -0.7451417446136475, -0.49162980914115906, -0.35426637530326843, -0.7616312503814697, 0.7666172385215759, -1.2642520666122437, -1.3454736471176147, 0.14998872578144073, -0.6899590492248535, 0.8958971500396729, -1.695914387702942, -0.17113836109638214, -0.7510635852813721, 0.3043096959590912, 0.34919846057891846, 0.40587347745895386, 2.2692179679870605, -0.9248857498168945, -0.08971057087182999, -3.207512617111206, -1.523072600364685, 1.3772679567337036, -1.139056921005249, -0.10739636421203613, -1.063360333442688, 1.1779060363769531, 0.5217437148094177, -0.1853305548429489, -0.6858181357383728, 0.39778003096580505, -0.4293351173400879, 0.1168833076953888, -0.8615283966064453, -0.4811691641807556, -0.34438756108283997, -0.1646472066640854, 0.831082284450531, 0.5477957129478455, -0.32519441843032837, -1.0981383323669434, 1.8928797245025635, -1.480573058128357, -0.7268821001052856, 0.27455586194992065, 0.5121231079101562, 0.2563215494155884, -0.019981587305665016, 1.7946614027023315, 0.02924930490553379, -0.883063018321991, 0.4870222508907318, 0.8093020915985107, 2.064472198486328, -0.17941808700561523, -1.3339718580245972, -1.2249305248260498, 0.22917385399341583, -0.8507964611053467, 1.8333686590194702, -0.006600635591894388, -0.4729629456996918, 1.1136888265609741, 0.10099123418331146, -0.969610869884491, -0.018815433606505394, -0.6849410533905029, -0.3552353084087372, -0.1880073845386505, 0.7789170145988464, 1.7034647464752197, -1.3575983047485352, 0.13745245337486267, 0.022662878036499023, -1.644256830215454, 0.1832721084356308, 0.1617550104856491, -1.1899409294128418, -1.2456541061401367, 0.35440731048583984, 1.7190430164337158, 0.13986359536647797, -1.7614455223083496, 1.858081579208374, 0.6057919859886169, 0.8813811540603638, -0.7600710988044739, -0.4852564036846161, -0.5564565658569336, 0.937732458114624, -0.891048789024353, -0.35490769147872925, -0.6257386207580566, -0.7055529356002808, 0.9707180857658386, 0.9919654726982117, -2.143411636352539, 0.1533747911453247, 1.0517754554748535, -1.5570533275604248, -0.9760533571243286, -0.6081719398498535, 1.0993667840957642, 0.5429257154464722, 0.4629082977771759, 1.0545750856399536, 0.031919267028570175, 0.5243192911148071, -0.6150535941123962, 0.8524320721626282, 0.04260104522109032, 0.26835450530052185, -0.9334573745727539, 1.7706044912338257, -0.29665520787239075, 0.7774854898452759, -0.6593642234802246, -0.9095449447631836, -0.05270830914378166, -0.4540320634841919, -0.859671413898468, 0.7482661008834839, -0.45404860377311707, 0.4599517583847046, 0.5295856595039368, -1.3410916328430176, -0.25043293833732605, -0.95243239402771, -0.10254815965890884, 0.3820425868034363, 0.5775129795074463, 1.1461070775985718, -1.2634989023208618, -1.6700067520141602, -0.004339444916695356, -1.1655040979385376, -0.20100006461143494, 0.6607092022895813, -1.0126968622207642, -0.2200070172548294, 0.6194667220115662, 0.6004123091697693, 0.07993879914283752, 0.41793060302734375, -0.6453203558921814, -0.832134485244751, 0.7293637990951538, 0.2903290390968323, -0.4470517635345459, 0.8772830963134766, 0.5749900341033936, -1.6430444717407227, 1.0322189331054688, -0.5312662720680237, -0.47535380721092224, -0.4241587221622467, 1.918279767036438, 0.9553952813148499, -0.10376535356044769, -1.1205121278762817, 1.1805713176727295, 0.11852063983678818, 2.4406049251556396, 1.159687876701355, -0.0954885482788086, -0.14781641960144043, 0.5428609251976013, 0.5989710688591003, -0.10022258758544922, -0.5760728716850281, 2.2747204303741455, -0.6276845335960388, -1.7118749618530273, -0.1447134166955948, -1.442807674407959, -0.6193989515304565, 0.9489777684211731, 0.8951398730278015, -0.13684406876564026, 1.865849256515503, 0.043197352439165115, 0.3826068043708801, 0.9713303446769714, 0.08913332968950272, 0.982540488243103, 0.385478138923645, -1.3249996900558472, -0.9473806619644165, 2.2256977558135986, 0.19476160407066345, 0.2532903254032135, 1.3071057796478271, 0.019482828676700592, 1.4439101219177246, 1.0660994052886963, -1.3803702592849731, 0.40645235776901245, -1.2219853401184082, -1.3133927583694458, -1.8705240488052368, -1.772557020187378, 0.25932687520980835, 1.1733065843582153, -0.3001657724380493, 0.4460877478122711, -1.2036056518554688, 0.35406404733657837, -1.0334311723709106, 0.11368907988071442, -0.9509204626083374, -1.5528266429901123, -0.14971399307250977, -0.2499794214963913, 1.1836647987365723, 0.5626001954078674, 0.3801344037055969, -1.879019021987915, -1.2448079586029053, -0.5904538631439209, -0.8133121728897095, -0.24572262167930603, 0.059582240879535675, 1.3676307201385498, 2.0663058757781982, -1.3789938688278198, -1.071967363357544, -0.30936500430107117, -0.32824382185935974, 0.7654819488525391, -0.07599832117557526, -0.6768179535865784, 1.6107970476150513, 0.4598514139652252, 1.181106686592102, 0.9096869230270386, -1.0836572647094727, 0.6626976132392883, 0.07446689158678055, -0.8070902824401855, 0.9076540470123291, -0.7288997769355774, -0.7920812368392944, -0.46875396370887756, 0.5340368747711182, 0.07368084788322449, 0.5980595946311951, 0.3280777037143707, -0.6266170144081116, 0.775101363658905, 0.2607073187828064, -0.5613733530044556, 1.009574294090271, -0.6999962329864502, -0.3150973916053772, -0.521250307559967, -0.04626719653606415, -1.2150487899780273, -0.6444159150123596, -0.33668574690818787, 0.046538304537534714, -0.494873970746994, 0.017106136307120323, -0.2557791471481323, -0.05588464066386223, 0.5136062502861023, 2.3433871269226074, 1.023667812347412, 0.13185255229473114, 0.5264800190925598, -0.8455483317375183, 0.0907009169459343, -0.4242604672908783, -1.1497867107391357, 0.8556376099586487, 0.4813239574432373, -1.6727728843688965, -2.178370237350464, -0.7442891597747803, -0.6825032234191895, -0.2809697389602661, 0.781769335269928, -0.3436437249183655, -1.0126057863235474, -1.6764391660690308, -0.3011411428451538, -0.7441110014915466, 0.33713123202323914, -1.3738183975219727, 0.6228223443031311, -0.5467215776443481, 1.217584252357483, 0.321021169424057, 1.2550431489944458, 2.4749503135681152, 0.8049946427345276, 1.7968729734420776, 0.26893600821495056, 0.18357115983963013, -1.0692036151885986, 1.0127547979354858, -0.9475050568580627, 0.1510748416185379, 1.8913531303405762, -1.4749747514724731, 1.6098694801330566, -0.21826186776161194, -0.361224889755249, -0.9444180727005005, 0.4350927174091339, -0.20081016421318054, -1.0232813358306885, 0.06624395400285721, -0.7919448018074036, 0.6467713713645935, -0.7806693911552429, 0.007700053043663502, 1.5048115253448486, 1.4736580848693848, 0.8911142945289612, 0.9672722816467285, -1.2149021625518799, -0.583202600479126, 0.9737988710403442, 0.06936189532279968, -0.9474908709526062, 0.17186684906482697, -0.34212610125541687, 0.25658780336380005, 0.3932150602340698, 0.9220918416976929, -1.9966894388198853, 0.6268378496170044, -1.1331291198730469, 0.9222488403320312, 0.9974473118782043, -1.2744802236557007, -0.1555166095495224, -0.8682230114936829, 0.9277876615524292, -0.6373891234397888, -1.481461524963379, -0.775892972946167, -1.5248959064483643, 0.21815238893032074, 0.8926955461502075, -0.6619595885276794, 0.049926649779081345, -0.7403881549835205, -1.3574533462524414, -0.8840076327323914, -1.7438029050827026, -0.07020515948534012, -0.27906450629234314, -0.6288007497787476, -1.0022684335708618, -0.24038590490818024, -1.1681816577911377, 0.44231685996055603, -1.4277552366256714, 0.7368882894515991, -0.075090691447258, -0.09991034120321274, 1.4433040618896484, 1.5442371368408203, 1.9780606031417847, -0.634472131729126, -0.6047881841659546, 0.25887271761894226, -0.16984033584594727, 0.6328117847442627, -1.013109803199768, 1.2328777313232422, -0.7677587866783142, 1.7141114473342896, -0.015544884838163853, 1.0116183757781982, -0.6032655835151672, -2.0087783336639404, 0.9004976153373718, -0.33225882053375244, -0.8001635074615479, -0.5932279825210571, 1.1640836000442505, -1.0974866151809692, 1.7911382913589478, -0.5747923851013184, -2.7817869186401367, -2.1826462745666504, -0.04946574941277504, -0.06361833214759827, 0.24390122294425964, 1.2144495248794556, -1.581000566482544, 0.13639840483665466, -1.1360516548156738, 0.5362324118614197, -0.22877632081508636, 0.020278405398130417, 0.24776825308799744, -0.49154484272003174, -1.723246693611145, -2.0885255336761475, 0.34069570899009705, -0.6609829068183899, 1.0215121507644653, 0.327258437871933, 0.5414977073669434, -0.5456803441047668, -0.3131674528121948, 1.202804684638977, 1.4269517660140991, 0.9874099493026733, 0.6403946876525879, 1.6906161308288574, 0.10087242722511292, 2.1842498779296875, 0.49325400590896606, 0.566605269908905, -0.11082510650157928, -0.3072972893714905, -0.011150014586746693, 1.0483061075210571, -0.9055478572845459, -1.4451028108596802, -0.23912747204303741, -1.3663607835769653, 0.8181779980659485, 0.7234128713607788, 0.27653366327285767, -1.5080549716949463, 0.7972080111503601, -0.49947258830070496, -0.4941425621509552, 0.15284405648708344, 0.09718897938728333, 1.1558548212051392, -0.39811959862709045, -1.7934839725494385, 1.0051586627960205, -0.5323801636695862, -0.42843157052993774, 1.324576735496521, 0.17355293035507202, -2.3804829120635986, 0.08459886163473129, 0.9881619811058044, -0.3281572759151459, 0.648474395275116, 0.3786123991012573, 0.12563398480415344, 0.14958250522613525, 1.0086169242858887, -0.44198763370513916, 0.04885543882846832, 0.292590856552124, 0.3975035846233368, -0.2936338782310486, 1.3296592235565186, -0.6484590172767639, -0.4565954804420471, -0.7808286547660828, -0.49178317189216614, -1.8583745956420898, 2.0311837196350098, -0.6244702935218811, -0.597099244594574, 0.9250009655952454, -0.37540972232818604, 1.7556147575378418, 0.8298038840293884, 0.7583067417144775, 0.7746995687484741, -1.6633504629135132, 1.0513026714324951, -0.16426660120487213, -0.34577393531799316, -1.6673604249954224, 0.839594841003418, 0.42193153500556946, -1.3864344358444214, -0.5339819192886353, -0.7893869280815125, -0.8016799092292786, 1.0024471282958984, -0.38780781626701355, -0.47802498936653137, -1.6606571674346924, -0.6882120370864868, -0.2954632341861725, -0.10955789685249329, 0.23077990114688873, -0.49641960859298706, -0.0782969519495964, -0.04107203707098961, 0.8069280385971069, 0.3400319814682007, -2.289883852005005, -1.2525428533554077, 0.3756929337978363, -1.759668231010437, -1.5605309009552002, 0.5221189856529236, 0.14325019717216492, 0.41961875557899475, 0.34768402576446533, -0.7204027771949768, 0.429180771112442, -0.0013955002650618553, 2.5859272480010986, -0.6839248538017273, 0.7318747639656067, 1.5920383930206299, -0.0017978395335376263, -0.5664280652999878, 1.0644539594650269, 0.38085803389549255, -0.41381749510765076, 0.778255045413971, -1.6620988845825195, 0.35695159435272217, 0.149557426571846, -0.3536002039909363, 0.20569610595703125, 1.182334303855896, -0.4289554953575134, 1.829598307609558, 0.23910591006278992, 0.11940986663103104, 0.528157114982605, 0.03926703706383705, 1.2250531911849976, -0.4873773455619812, 2.060880422592163, 0.4167450964450836, 0.9451004862785339, 0.587464451789856, 1.2192890644073486, -0.7687619924545288, -0.9683747291564941, 0.9747982621192932, 0.45166701078414917, 1.8051739931106567, -0.4419962465763092, -0.7722108960151672, 0.1849851906299591, -1.466605544090271, -0.13872677087783813, -1.2211134433746338, -0.053514521569013596, 0.6934595108032227, 0.43992483615875244, 0.08397597074508667, 0.41483011841773987, -1.5840295553207397, 1.0538910627365112, -1.2888429164886475, -1.3719664812088013, 1.4868743419647217, -0.9312632083892822, 0.2709242105484009, 0.15188729763031006, 0.34550973773002625, -0.419857382774353, 1.3704715967178345, 2.0634093284606934, -0.3052448630332947, -0.21869798004627228, -0.6420319676399231, 0.02239154279232025, 1.2329081296920776, 0.4155612587928772, 0.8467637300491333, 0.09410400688648224, 1.2811026573181152, -1.0262426137924194, 0.7719026803970337, 0.9141406416893005, 0.49368393421173096, 0.24849800765514374, -0.8502700328826904, 0.1006002351641655, -0.45017412304878235, 1.4581716060638428, -2.0511560440063477, 0.045427847653627396, 0.2681823968887329, 0.5755741596221924, -0.5628368854522705, -0.46255847811698914, 0.4121478199958801, 1.093855381011963, 0.14356930553913116, 1.1939669847488403, -1.1150346994400024, 0.9124937653541565, 0.627439558506012, -0.2251099944114685, -1.5981783866882324, 0.9599329233169556, 0.9163327217102051, 0.7744393944740295, 0.9302804470062256, -1.1940655708312988, -0.03612317517399788, 0.4398912787437439, -0.275172621011734, 1.0518231391906738, 1.940915584564209, 0.4685429334640503, -0.3700107932090759, 0.004010469187051058, 1.1853134632110596, -1.2316852807998657, -0.6681257486343384, 2.5672378540039062, -1.020674467086792, -1.2380319833755493, -0.43492329120635986, 0.30763188004493713, -0.5364293456077576, 0.43996843695640564, 1.0979682207107544, 0.050424110144376755, -1.0523356199264526, -0.8227348923683167, -0.16756127774715424, -0.1027020812034607, -1.8235710859298706, 0.9478851556777954, -0.16167686879634857, -0.39938244223594666, 0.025625428184866905, -0.6337050199508667, -0.3719829320907593, -0.8679211139678955, 0.3303878903388977, 1.6770656108856201, -0.7298445105552673, -0.6515716314315796, 0.40725332498550415, -1.0787694454193115, 0.051091790199279785, 0.9104883074760437, 1.3314231634140015, 0.4635726809501648, 0.30441537499427795, 0.43980440497398376, -0.5452417731285095, -1.2402663230895996, 0.05060647055506706, 0.26772844791412354, -1.67497718334198, 0.1855403333902359, 1.2067409753799438, 0.47692790627479553, 1.2564667463302612, -0.6401493549346924, -0.8737976551055908, 1.2524917125701904, 1.2194271087646484, -1.1815283298492432, -0.014009413309395313, -1.318235993385315, 1.2658052444458008, 0.35719388723373413, -0.22140274941921234, -0.3980870842933655, 0.7107534408569336, -0.6404269933700562, 0.724414587020874, 1.1319485902786255, -0.9323193430900574, -2.309373140335083, 0.5706270337104797, 0.05217508599162102, 0.4635683000087738, 1.1937766075134277, -0.7260891199111938, -1.5688447952270508, 0.2388913929462433, 0.5648783445358276, -0.016464976593852043, -1.7329246997833252, 1.1620395183563232, -0.2874261140823364, 0.8848943710327148, -0.0986393615603447, 0.010756928473711014, -0.6714580059051514, -0.7211576700210571, -0.1938932240009308, 0.11323235929012299, -0.26232442259788513, -1.8012081384658813, -0.9011401534080505, 1.4502148628234863, -0.30794239044189453, -1.0049203634262085, 1.0867747068405151, 0.8830381035804749, 1.1841518878936768, -0.6775807738304138, -1.7622581720352173, -2.0146901607513428, 0.9437726736068726, -0.19217364490032196, 0.3156052529811859, -1.2669742107391357, 0.12710431218147278, 0.18987062573432922, 0.261599600315094, -0.40058231353759766, 0.13727161288261414, -0.4418421983718872, 0.6787495017051697, -0.2677631676197052, 0.6786053776741028, 0.18999625742435455, 1.7043570280075073, 0.4364246726036072, -0.8197593688964844, 0.7884529829025269, -0.9493951797485352, 0.43751829862594604, 0.5085899233818054, 0.43398624658584595, 1.3640159368515015, -0.7553938031196594, 0.685678243637085, -0.5500068068504333, -0.14208820462226868, -0.6519917249679565, -0.9051415324211121, -0.17293359339237213, 0.1757926642894745, -0.09187721461057663, -2.129333257675171, -1.326273798942566, 1.245341181755066, 0.8171681761741638, -0.7617323994636536, 0.12847305834293365, -0.02326984889805317, -0.18967129290103912, 0.841844916343689, 0.4903021454811096, -0.3354058861732483, 0.13235998153686523, 0.5633506178855896, 0.05222097784280777, -0.21902675926685333, 1.9153389930725098, -0.5441929697990417, 0.5747820734977722, 0.04140635207295418, 1.068839430809021, 0.6591048836708069, -0.5365167856216431, -2.0676472187042236, -0.05743655189871788, 0.5425443649291992, 0.434007465839386, 1.3428784608840942, -0.28603479266166687, -1.2325294017791748, -0.4902023375034332, -2.1708483695983887, -0.8177127242088318, -0.9154534935951233, -0.0010428413515910506, -1.0546754598617554, -0.9632186889648438, -0.4512522518634796, -0.7126433253288269, -0.7884514331817627, 0.09846580773591995, -0.9309483766555786, -1.2912335395812988, -0.4027734696865082, -0.13128304481506348, -0.4000554084777832, 1.3350199460983276, 1.5046659708023071, 0.6560202240943909, -0.5902998447418213, 0.8299421072006226, 0.6837029457092285, -0.4958321154117584, -1.9817478656768799, 0.4253738522529602, 0.44806262850761414, 0.4682157635688782, 0.7076467275619507, 1.189850926399231, -2.0900585651397705, 0.5095437169075012, 0.39230257272720337, -0.12198641896247864, 1.477510690689087, -0.16073189675807953, -1.3871623277664185, 1.3517166376113892, 0.7523610591888428, 0.3785487115383148, -1.6613962650299072, -1.1519986391067505, -0.8790980577468872, 0.20620742440223694, -0.3527149260044098, 2.288848876953125, 0.1643276959657669, -0.7993689775466919, 0.27824488282203674, -0.5047948360443115, 0.847034215927124, -0.22096805274486542, -0.06631728261709213, -0.722791314125061, 0.08947734534740448, -1.5160253047943115, -0.47786590456962585, 1.3525887727737427, 1.066241979598999, -0.5999175310134888, 0.7091066837310791, 0.0959138572216034, 0.43401506543159485, 1.5125972032546997, -0.4617023766040802, 1.970403790473938, 0.20784559845924377, -0.7668614983558655, 0.7969279289245605, -0.9567078948020935, 0.64327472448349, 0.8596223592758179, 0.07010751962661743, -1.7105002403259277, 2.101933002471924, -0.04177483543753624, -1.2326712608337402, -0.30898040533065796, -0.08087078481912613, 0.3129062354564667, 0.05030133202672005, 0.22515784204006195, 0.20665135979652405, 1.094208002090454, -0.9232989549636841], [-0.1510026454925537, -1.5579602718353271, 1.688238263130188, 2.638911008834839, -0.8483965396881104, -0.23225250840187073, -1.2788959741592407, 0.4037303328514099, 1.9749150276184082, 1.2080357074737549, 1.0567866563796997, -1.292754054069519, 0.9761786460876465, 0.08628489822149277, -0.7598533034324646, -1.0807281732559204, 0.6424659490585327, 1.0060898065567017, -0.36867421865463257, -0.1611565202474594, 1.3092999458312988, -0.004924225155264139, 0.4392088055610657, -2.1653268337249756, -0.04642599821090698, -0.10436756163835526, -0.3282022476196289, 0.8673107624053955, 0.664729654788971, -0.391048401594162, -0.03746546804904938, -1.4635752439498901, -1.5883592367172241, 0.9705421924591064, 0.2084423005580902, 1.8154146671295166, -0.7655755877494812, -0.8073493838310242, -0.4766244888305664, 1.300502896308899, 0.3290177881717682, -1.506780743598938, -0.7447265982627869, -1.206922173500061, -0.56891268491745, -1.3495017290115356, -0.41883838176727295, -0.6872984170913696, 1.1966149806976318, 1.019004464149475, 0.8146271705627441, 0.7007436752319336, -0.6905491948127747, -0.30454277992248535, 0.7198513746261597, -1.8436095714569092, 0.21374844014644623, -1.6708608865737915, 0.8209761381149292, -2.633253812789917, 0.3427455723285675, 0.6133553385734558, -0.3782919943332672, 0.1446627974510193, 1.2198470830917358, -2.0170886516571045, -1.2187755107879639, 1.7436667680740356, -0.21752846240997314, -0.6279292702674866, 0.7505509853363037, -0.8887926340103149, 0.23425766825675964, -0.8688222765922546, -1.9924280643463135, 1.0920876264572144, -0.6737642884254456, 1.0891709327697754, 0.5204585194587708, -0.392843633890152, 0.29522040486335754, -0.11683652549982071, 0.02569272369146347, -0.8792780637741089, 0.8326948285102844, 1.2221864461898804, 0.6680082678794861, 0.7017308473587036, -1.5650835037231445, -0.11583494395017624, -0.511837363243103, -0.22140011191368103, 0.08657998591661453, 0.10887299478054047, -0.7838290929794312, 0.9093474745750427, 0.8512073159217834, -0.6181463003158569, -1.768631935119629, 0.8004680871963501, -0.48394137620925903, 1.6282044649124146, 0.033529989421367645, 1.4160698652267456, 0.15372854471206665, 2.0372347831726074, 1.3775399923324585, 0.22461958229541779, 0.34549233317375183, 0.2169383466243744, 0.3530373275279999, -1.9390511512756348, -0.4073741137981415, 0.3713483512401581, -0.06954086571931839, -0.6352640986442566, 0.005181078799068928, 0.6029266119003296, 0.7903007864952087, 0.49808409810066223, 1.4087289571762085, 1.1439690589904785, 1.64272940158844, -0.07800181955099106, 0.8612290024757385, -0.34400931000709534, -0.9556150436401367, 0.10635942965745926, 0.6057844161987305, -1.270613431930542, 0.5728284120559692, -0.1690652072429657, 1.8272066116333008, -2.0063092708587646, -0.4923383295536041, 1.5097798109054565, 1.9044970273971558, -0.2619744539260864, 0.10911895334720612, 1.2862716913223267, -1.4581983089447021, -0.3176257908344269, 0.41787680983543396, 0.5305943489074707, -0.025482412427663803, 0.750307559967041, 0.9843029975891113, 1.5869969129562378, 1.3685975074768066, 1.5405733585357666, -0.85821932554245, 1.0468202829360962, -0.8454352617263794, 0.45330381393432617, -0.8811311721801758, 0.8595916628837585, -0.4575073719024658, -2.274657726287842, -0.22826458513736725, 0.30376094579696655, 0.24083209037780762, -1.5089644193649292, 0.8102943301200867, -1.4131439924240112, -1.1337110996246338, 0.06061595305800438, 1.0721137523651123, 1.8167465925216675, 0.2217860072851181, -0.21459731459617615, -1.0002353191375732, -0.5611628293991089, -1.3640437126159668, 0.21374236047267914, -1.388861060142517, 0.22737489640712738, -0.5063661932945251, -0.8867278099060059, -0.07595793157815933, 1.7805784940719604, -1.294051170349121, 0.09808609634637833, 0.20621497929096222, 0.6580412983894348, -1.1746681928634644, -0.9416972398757935, -0.727411687374115, 0.6821761727333069, -1.4900646209716797, -1.064317226409912, -0.16694001853466034, 0.8636266589164734, -0.09823378920555115, 0.8618854880332947, -0.580191969871521, 1.1360931396484375, -0.6649913191795349, -1.1299105882644653, 1.9437240362167358, 0.11905109137296677, 0.18707777559757233, 0.6613572835922241, 0.9526653289794922, -0.2536080479621887, -0.8859041929244995, -0.088448166847229, 0.22809968888759613, -0.347372442483902, 0.372009813785553, 0.8752883672714233, -0.12778283655643463, -0.5234977602958679, -0.08584526926279068, -0.22341381013393402, 0.9819095134735107, -0.45593056082725525, -0.56001877784729, 0.1951419860124588, 1.3654506206512451, -1.6906938552856445, 0.8792094588279724, 1.261716604232788, -0.3945760130882263, 0.5000556707382202, 1.775033950805664, -0.029943961650133133, 1.4065901041030884, -0.8156198859214783, -0.20380036532878876, 1.6741987466812134, 1.434606909751892, -0.15370315313339233, 0.4350069463253021, -0.6176952719688416, 0.4206774830818176, 0.5815973281860352, -0.4397290050983429, 0.9494333863258362, 0.6775388717651367, 2.974836587905884, -0.22673024237155914, -1.14849853515625, 1.2232707738876343, -0.6998704671859741, 0.6385998725891113, -1.0379544496536255, -0.2755029797554016, 0.40089765191078186, -0.4551946222782135, -0.4154182970523834, 0.469186007976532, -0.1377672255039215, 1.4978266954421997, 0.5704386830329895, -0.4366443157196045, -0.3370690643787384, 1.6415960788726807, -0.3583414554595947, -0.3629429042339325, -0.23178991675376892, -0.1557089239358902, -1.3344030380249023, 2.908147096633911, -1.7842555046081543, -0.9052008390426636, 1.3595850467681885, -0.878505527973175, 0.19394376873970032, -1.1380112171173096, 0.6235449910163879, 1.2974038124084473, 0.6372585892677307, 1.3101898431777954, -0.5460366010665894, -0.010521147400140762, 1.143881916999817, 0.09361925721168518, -0.19484853744506836, 0.9758894443511963, -0.026429714635014534, 0.7525730133056641, 0.022042717784643173, 1.04361093044281, -0.497879296541214, 0.3896750509738922, -0.12325863540172577, -0.1496245414018631, 0.3785926103591919, 1.5143764019012451, 1.0802255868911743, 1.4021159410476685, 0.8714120388031006, -0.43489575386047363, -0.22517742216587067, 1.1960620880126953, 0.5864790081977844, -0.3358674645423889, -2.0830061435699463, -0.058346234261989594, 2.1117804050445557, -0.19715945422649384, -0.5829497575759888, -0.2557761073112488, 0.6189051270484924, 0.3923601508140564, -0.9136074781417847, -1.4557828903198242, -0.2493431121110916, -0.9534491300582886, -1.4081676006317139, -0.8751765489578247, -0.4792802333831787, -0.5632444024085999, -2.153446912765503, 0.1099340170621872, -0.46095216274261475, -0.4388364553451538, -1.4662961959838867, 0.007793933618813753, -1.0215553045272827, -0.36662557721138, -1.2230076789855957, 0.4627828896045685, -0.6412953734397888, -0.20698712766170502, -0.5590264797210693, 0.7939272522926331, 0.0711301788687706, -0.1875627189874649, -0.770828902721405, 0.5384693145751953, 0.40107885003089905, -0.4148973524570465, 0.05519067868590355, 1.0126949548721313, -0.8903892040252686, -0.6057053804397583, 0.7637063264846802, 1.350928544998169, -0.9747885465621948, -1.0236501693725586, -1.1993621587753296, -0.7510678172111511, -0.07523448765277863, -1.759758472442627, -0.5084441900253296, -1.1681216955184937, 0.16470354795455933, -1.0538017749786377, -1.9471780061721802, -1.4189343452453613, -1.081169843673706, -0.40681537985801697, 0.273040771484375, -1.5369032621383667, -0.0987263023853302, -0.07605704665184021, 0.19653522968292236, 1.2013672590255737, 0.07253773510456085, -0.7249956727027893, 0.0840473398566246, -0.6864510178565979, 0.009304467588663101, -0.6910257339477539, -1.5351027250289917, 2.4307608604431152, -1.7856332063674927, -0.7544872164726257, 0.09352316707372665, 0.1816260665655136, -1.9140108823776245, -0.9316338896751404, 1.2291312217712402, -1.8525259494781494, 2.6433706283569336, 0.3249348998069763, 1.2276145219802856, -1.82939875125885, -1.1270664930343628, 2.0292961597442627, 0.027299530804157257, -1.2001748085021973, 1.4783276319503784, 0.5140249729156494, 0.5903297662734985, 0.480090856552124, -0.10988704115152359, 0.004429392516613007, 1.388632893562317, -1.256887435913086, 0.9103708863258362, -0.5219656825065613, 0.03526000678539276, -1.137622594833374, -0.3940906226634979, -1.1433018445968628, 1.158538579940796, 0.49513357877731323, 0.6531323790550232, -0.9803575277328491, 0.3266558349132538, 0.16592298448085785, 1.2499232292175293, 0.28447961807250977, -0.19705498218536377, 0.6175702214241028, -0.10982359945774078, 2.043736457824707, 0.10347923636436462, -1.3964097499847412, 0.004574727267026901, 1.0876944065093994, -0.05435263738036156, 1.102325201034546, -0.6504620313644409, -1.8906869888305664, 0.9145770072937012, -0.3219240605831146, 1.531602382659912, -0.9976658225059509, 0.11364378035068512, -0.10245699435472488, 1.5503227710723877, 1.7739174365997314, -0.9521788358688354, 0.26931774616241455, 2.245239019393921, 1.680583119392395, 1.0572446584701538, -0.36378467082977295, 0.0412931963801384, -0.29488393664360046, 1.47433602809906, 2.07515549659729, 0.37182748317718506, -0.5438777804374695, -1.9212660789489746, -0.1087665855884552, 0.6943404674530029, -0.4504345953464508, -0.3012544810771942, 0.5510774254798889, -1.3884888887405396, 0.5012521147727966, -0.5082690119743347, -0.03496323525905609, -0.6416751146316528, -0.12777045369148254, -1.8183733224868774, 0.12311538308858871, -1.3123605251312256, -1.1076759099960327, -0.4022707939147949, -0.2890877425670624, 0.4292002022266388, 0.7922412157058716, -0.013191498816013336, 1.344009280204773, 0.07119134813547134, 0.9899333715438843, -0.21665029227733612, 2.6682000160217285, -0.3177676200866699, -1.554168701171875, -0.12948495149612427, -0.5194068551063538, 0.6249819993972778, 0.06657426059246063, -0.4107367992401123, 0.7007293701171875, -2.456249237060547, -0.3645244836807251, -0.7672946453094482, 1.2389329671859741, 2.8344357013702393, 0.5858023762702942, 1.1817195415496826, -0.8094140291213989, -0.09665005654096603, 1.18746018409729, 0.8060880899429321, -0.5102764964103699, 0.4369199872016907, -0.43262621760368347, -0.8616697788238525, 0.6857621669769287, -0.7260834574699402, -0.4957655072212219, 1.3433300256729126, 0.45390304923057556, 0.3215906620025635, 2.9196996688842773, 1.5695382356643677, 0.4451570510864258, -1.8253310918807983, -0.7674903273582458, -0.07265531271696091, 2.933063268661499, 0.28400418162345886, -0.2503015697002411, -0.8293130397796631, 1.425572395324707, 0.6473782062530518, -0.2154400646686554, -0.5904046297073364, 1.816908836364746, 0.12528125941753387, -2.0460147857666016, 0.016046054661273956, -1.414137840270996, -1.2215888500213623, -1.0976160764694214, -0.5653725266456604, 0.11409373581409454, -0.019565952941775322, -0.3515187203884125, -2.352576732635498, 0.8610659837722778, -0.22876299917697906, 2.3359646797180176, 1.1693087816238403, -0.25595560669898987, 0.9141282439231873, -0.6911080479621887, 0.71046382188797, 0.9933344125747681, -2.037278413772583, 0.03969082981348038, 0.6861965656280518, 0.21017740666866302, 0.41285407543182373, -0.12860393524169922, -1.8028264045715332, -0.29433879256248474, -1.9761905670166016, 0.6173753142356873, 1.8954503536224365, 0.2466905564069748, -1.2098957300186157, -2.2253856658935547, -1.2927898168563843, -1.6947882175445557, -0.3261524438858032, -0.8331778049468994, 0.697422981262207, 1.1757036447525024, 0.4001283645629883, 0.34835654497146606, -1.4672430753707886, -1.1953705549240112, -1.1650785207748413, 0.5463181138038635, 0.46430861949920654, -1.4628064632415771, -1.300722360610962, 0.4001500904560089, -0.7094717621803284, 1.4277452230453491, 0.806144654750824, -0.9981366395950317, 0.9200560450553894, -0.6613667607307434, 1.582200050354004, -0.7093362212181091, -1.308618426322937, 0.4451979994773865, 2.5272793769836426, -0.28037962317466736, -1.2074671983718872, -0.49824443459510803, -0.39094090461730957, 0.45113712549209595, 0.001586016034707427, 0.3437706530094147, 0.7168686985969543, -0.11799465119838715, -0.3790974020957947, 0.47882357239723206, 1.0821439027786255, 1.0842657089233398, -0.09710562974214554, -0.3489161729812622, -0.11311405897140503, -0.8248390555381775, 0.27578508853912354, -1.2863476276397705, 0.5566222667694092, 0.6069117784500122, -0.4891873896121979, 1.3811540603637695, -2.0676677227020264, -2.0572926998138428, -0.06461689621210098, -0.06352859735488892, 1.2784199714660645, -2.0490095615386963, -0.8737157583236694, 0.5149068236351013, 1.6897997856140137, -1.173270583152771, -1.624672532081604, -1.8920512199401855, -0.020302902907133102, -0.6432525515556335, -0.2094174474477768, -0.1099824458360672, 0.06683558970689774, -0.21604812145233154, -0.5476104021072388, -1.0810177326202393, -0.9928067326545715, -2.064640998840332, -1.0319405794143677, -0.17218747735023499, 0.12814851105213165, 0.14241613447666168, 0.14071930944919586, -0.47381144762039185, 0.1167832463979721, 0.7863100171089172, 0.6063106656074524, -0.043953076004981995, -0.819581925868988, 1.8079725503921509, 1.9139509201049805, -0.6732597351074219, -1.322318434715271, 0.2251487672328949, 1.11436927318573, -0.00338192330673337, -0.7808920741081238, 0.2243892252445221, 0.529773473739624, -1.3851231336593628, -1.0298579931259155, -0.5023612976074219, -1.67270827293396, -0.6585215926170349, 1.754460096359253, -0.6557913422584534, -2.4154932498931885, 0.47443148493766785, -1.058082103729248, 0.2881205379962921, 1.0917346477508545, -0.48600107431411743, 1.5243539810180664, -0.1633153259754181, 1.7611156702041626, 0.7564290165901184, 0.9787254333496094, -0.3057834804058075, -0.21732084453105927, 0.6405547261238098, 0.20466205477714539, -0.09162954241037369, 1.1860445737838745, 0.5932387113571167, 0.1183776780962944, -0.3487042784690857, -0.9123703837394714, -0.09284339100122452, -0.6633790731430054, 0.9360988736152649, 0.3790189325809479, -2.0626115798950195, -0.7399746775627136, -0.4878770709037781, 0.3726966679096222, 0.03485564887523651, 1.119924783706665, 0.4412990212440491, -1.3123445510864258, -0.29874467849731445, 1.5032095909118652, -0.6363258361816406, -0.7797728776931763, 0.815609335899353, 1.0494321584701538, 0.5204365849494934, 0.9206374883651733, -0.7273755669593811, 0.33098816871643066, -0.4684165418148041, -0.1424168199300766, 1.2088816165924072, 0.035384275019168854, -0.2585844099521637, 0.581575870513916, 0.5138956904411316, 0.0425446555018425, 0.8699244260787964, -0.11845297366380692, -0.07997602224349976, 0.02241721749305725, -0.2791454792022705, -0.11457107961177826, -1.28438401222229, -0.3911871910095215, 0.18472833931446075, 0.1969088315963745, -0.5640827417373657, 0.1645108312368393, -1.3732383251190186, -1.2143619060516357, 0.17804847657680511, 0.5070430636405945, -0.07233188301324844, 0.2696970999240875, -0.7281171083450317, 0.23456579446792603, 0.6996700167655945, 1.283643364906311, -1.3004002571105957, 0.034537579864263535, -0.03516869992017746, 0.9562426209449768, 0.1792590469121933, 1.255705714225769, 2.660686731338501, 0.17224928736686707, -1.1066447496414185, 0.9471834301948547, -0.21191047132015228, -0.8153905272483826, 0.28852275013923645, -0.5452606081962585, 0.6604920625686646, -1.0510717630386353, 0.08352772146463394, -0.11313122510910034, 1.5204391479492188, 0.8606997132301331, 1.4601606130599976, -0.6823508143424988, -0.7634825706481934, 2.0554356575012207, 0.6574505567550659, -0.4239671230316162, -0.8494023680686951, 0.50398188829422, -0.866814374923706, 0.07660361379384995, 1.0821481943130493, 0.6329649686813354, -0.7650405168533325, -1.2042710781097412, 0.4803631901741028, 0.05449360981583595, -1.4692667722702026, 0.4494084119796753, 0.9331848621368408, 0.29945117235183716, 1.719084620475769, -0.2974992096424103, 0.3833840489387512, -0.8422135710716248, -1.0905743837356567, -0.32171186804771423, 1.3080354928970337, -0.12660109996795654, -0.7331914901733398, -0.18090398609638214, 0.5183642506599426, -1.0228456258773804, 1.4703398942947388, 0.6585819125175476, -0.334863543510437, 0.48353227972984314, -0.4014159142971039, -0.1805671900510788, -1.0837081670761108, 1.217240571975708, 0.6439228057861328, 0.11830034852027893, -0.2631058990955353, -1.016413688659668, 0.15590712428092957, 0.47691452503204346, 0.020568756386637688, 0.5202595591545105, -1.7846999168395996, -0.7323245406150818, 1.067116141319275, -0.7416960597038269, -1.1353025436401367, 0.06387953460216522, 0.9950998425483704, -1.4021682739257812, 0.287226140499115, 0.40281495451927185, 0.9683154821395874, 1.6173341274261475, -0.8128886222839355, 0.41080302000045776, -1.5434097051620483, -1.1177310943603516, -0.05023735761642456, 1.2616190910339355, -0.4313298761844635, -0.538070559501648, 1.1773855686187744, 1.4706717729568481, 0.7947892546653748, -0.06154872104525566, -0.1908665895462036, -0.387066513299942, 2.05163311958313, -0.2638414204120636, -0.3894297182559967, -0.6820738315582275, -2.0758399963378906, 1.8960120677947998, 0.5234455466270447, 0.4980073571205139, 0.46180459856987, -0.4128211438655853, 0.3517202138900757, -1.1837764978408813, 0.11899732798337936, 0.9669784903526306, 0.6152737140655518, 0.2892334461212158, -2.1715917587280273, 0.6462644934654236, -0.12748590111732483, -0.5774750113487244, 0.22350959479808807, -0.34155017137527466, -2.058171033859253, 1.649425983428955, 0.36071476340293884, -1.0034737586975098, -0.603469729423523, -0.5607550740242004, -2.248414993286133, -0.5859776735305786, 0.629937469959259, 1.210283875465393, -0.7700514793395996, -0.4747937023639679, -0.626477837562561, -0.6935461759567261, 1.1478732824325562, -0.6295035481452942, -0.5060538053512573, -0.5902933478355408, -0.3419669568538666, -0.054123569279909134, 1.3338978290557861, 0.4649420976638794, 0.07161805033683777, 1.4640171527862549, -0.8131410479545593, 0.4009989798069, -1.4154881238937378, -1.7977159023284912, -0.6819786429405212, 0.6708715558052063, -0.6822171807289124, 0.8807506561279297, 1.313615083694458, -0.36739009618759155, 1.3838070631027222, 0.09697308391332626, -0.754467785358429, -0.7005911469459534, 0.7091996073722839, 1.2416954040527344, -0.11872369796037674, -0.8817152380943298, 0.3519894778728485, 1.133945107460022, 0.4696177542209625, 1.079649567604065, 0.0727134570479393, 1.4126386642456055, 0.8550906777381897, -0.09783712029457092, 0.36386623978614807, 3.256844997406006, 1.4798145294189453, -0.14712285995483398, 2.3263280391693115, 0.3838881850242615, 1.523950457572937, 0.8593214154243469, -3.0188262462615967, -0.9012177586555481, 1.7524075508117676, 0.8039433360099792, -0.38234204053878784, -0.45653265714645386, -1.9667022228240967, -0.8353530168533325, -1.5580458641052246, 1.3847352266311646, -1.5032697916030884, 0.45362189412117004, -0.38428857922554016, 1.506433129310608, -1.3418002128601074, 0.5905253887176514, 0.09479373693466187, 0.4111109972000122, -0.3569658398628235, -0.5920321345329285, -1.2482928037643433, 0.5262984037399292, 0.019004058092832565, 0.775315523147583, -2.312131643295288, -0.17912520468235016, 1.5153132677078247, 0.39460742473602295, -0.8924567103385925, -1.8963143825531006, -1.0075888633728027, 0.9224603772163391, 1.852342128753662, -1.3207306861877441, 0.3556535542011261, 0.7532927989959717, 0.05904167890548706, 2.0661747455596924, 0.3952716588973999, 1.0452810525894165, 1.0902278423309326, -0.06697548925876617, -0.03486837074160576, -0.6325917840003967, 0.2910546064376831, 0.45488911867141724, -0.29441630840301514, 0.18151754140853882, -0.18365533649921417, -0.30447643995285034, 0.32630062103271484, -0.10676825791597366, -0.5184924006462097, -0.6856380701065063, -0.18454128503799438, -0.32477006316185, -1.2195318937301636, -0.2529943883419037, -0.7004973292350769, 0.30472472310066223, 0.5029218792915344, 0.599940299987793, -1.7544869184494019, 1.2196935415267944, 1.1981465816497803, -1.2428864240646362, 1.3261094093322754, 2.5758111476898193, 0.45594003796577454, -1.60117506980896, 1.761955738067627, -1.0607401132583618, -0.7407718300819397, -0.14376366138458252, -0.8015544414520264, -0.22094625234603882, -0.8238250017166138, -0.36099928617477417, 1.3205252885818481, -0.12310483306646347, -0.40763041377067566, -0.7744712233543396, -2.030757188796997, -0.1638832986354828, -1.2098463773727417, -0.890728771686554, -1.1741697788238525, -1.3555288314819336, -1.8041155338287354, -0.3410564363002777, 0.4164862036705017, -1.536049485206604, 0.26449576020240784, -0.9559063911437988, 0.7753862142562866, -1.1502423286437988, 0.9078613519668579, 0.4290933907032013, 0.05516088381409645, 1.4368152618408203, 0.26100024580955505, -0.39959263801574707, 1.9210625886917114, -0.5802270770072937, 0.15729747712612152, -0.47382980585098267, 0.4679211378097534, 0.4047900140285492, -0.6275011301040649, 0.3586379885673523, -1.0282760858535767, -0.7905542850494385, -0.2179950326681137, -1.894338846206665, -0.45704516768455505, 2.5730981826782227, -0.5523237586021423, 0.022027816623449326, -0.48598015308380127, 1.3029958009719849, -0.14224480092525482, -0.5420292019844055, -0.6903041005134583, 0.967460572719574, 0.07452594488859177, 0.40541622042655945, -0.520548403263092, 0.9080207347869873, 0.26922884583473206, -0.32060039043426514, -0.9963779449462891, 1.1217314004898071, -0.8797710537910461, -0.054069388657808304, -0.1626654714345932, 2.1357133388519287], [-0.17058022320270538, -0.2950335144996643, -0.6938586235046387, 0.30928266048431396, -1.2930938005447388, -0.29269173741340637, -1.0052186250686646, -0.10668487101793289, -1.070707082748413, -0.27973857522010803, 1.0068398714065552, -1.3748629093170166, -0.7600460648536682, -1.6326099634170532, 0.3609676957130432, 1.8324456214904785, 0.7683724761009216, 0.013103297911584377, -0.05031990632414818, 0.32294389605522156, 1.7637311220169067, -1.6419010162353516, 0.35066384077072144, 1.0016276836395264, 0.24671852588653564, -0.5201197266578674, -0.3151017725467682, 1.7295455932617188, -0.4400844871997833, -0.9534692764282227, 0.7987741827964783, -1.1665948629379272, -0.23057769238948822, -0.3487577438354492, -0.6604213118553162, 0.06460270285606384, 0.8363156318664551, 0.13492342829704285, 0.9375556111335754, 1.4709153175354004, -0.04748977720737457, -0.8578892350196838, -1.276044249534607, 2.8214571475982666, 1.9576547145843506, 0.4701649248600006, -0.6241200566291809, -0.4253831207752228, 0.10858684778213501, 0.3929581940174103, -0.4029317796230316, 0.4984404146671295, -0.001176095800474286, 0.2542344033718109, -2.0090534687042236, 0.8887133002281189, -0.4091901481151581, 0.7532919645309448, 1.1161116361618042, 1.1850395202636719, -0.5026103854179382, -0.4581449329853058, 0.07855736464262009, -0.26070737838745117, -0.7201637029647827, 0.21930184960365295, -0.07473020255565643, 2.293503999710083, 0.17414477467536926, -1.0165374279022217, 0.8225675225257874, 0.050089772790670395, -0.48208436369895935, -0.9851522445678711, -0.855242908000946, -0.35188719630241394, -1.0560390949249268, -0.3501286506652832, -1.6016830205917358, 2.085545778274536, -0.19242487847805023, -1.4810271263122559, -0.045457374304533005, -0.3464255928993225, -0.34184226393699646, -0.9182388782501221, -0.3736555278301239, -0.3044167160987854, -0.2079201191663742, -0.42180758714675903, 3.4942097663879395, -0.2722721993923187, -1.022856593132019, 1.2162542343139648, -0.6810707449913025, -0.10173799842596054, -0.9287946820259094, -0.7269009947776794, -0.49994534254074097, 1.4902020692825317, 1.1569486856460571, -1.725191354751587, 1.1324830055236816, 0.8143566846847534, -0.10295019298791885, 0.5295102596282959, -0.3392699956893921, 0.6057435274124146, 1.2393299341201782, -0.5781558752059937, 0.24106182157993317, 0.04615945369005203, 0.09556973725557327, -0.10980013012886047, -0.05165461078286171, -0.2671603560447693, -0.4787180721759796, -0.13818512856960297, 0.29181063175201416, -1.094806432723999, 0.09793077409267426, -0.41439899802207947, -0.7990633845329285, 0.7171269059181213, -1.7037025690078735, 0.3014333248138428, -0.013698603957891464, -2.290778875350952, 0.7404521107673645, -0.5437893867492676, 1.3086267709732056, 0.8467197418212891, -1.455928921699524, -0.2224052995443344, -0.12133020162582397, 0.8004744648933411, 0.3106556832790375, 1.0441484451293945, 0.9927957057952881, -1.7706701755523682, -0.09328410774469376, 0.3230050206184387, 0.9497881531715393, 0.8869262933731079, 0.006817181594669819, 2.4614486694335938, -0.6442245841026306, 0.31019970774650574, 1.2397170066833496, 1.0238441228866577, 0.050949763506650925, 0.8489552736282349, -1.279154658317566, -1.5115631818771362, 0.9522383213043213, 0.6289249062538147, 1.5309442281723022, -1.1037081480026245, -1.1685750484466553, -1.410689115524292, 0.27759310603141785, -0.7045177817344666, 0.6032496690750122, -0.017336973920464516, 2.0226187705993652, 0.2708090841770172, -1.078178882598877, -0.7581645846366882, 0.8127248287200928, -0.6593065857887268, 0.18671484291553497, -1.3553777933120728, 0.30936792492866516, 0.6922856569290161, -0.7061076164245605, 0.2156372219324112, 0.08083318173885345, 1.0293411016464233, -0.4502805769443512, 0.616668164730072, -1.6321866512298584, 0.5381054282188416, -0.2568763196468353, -0.38415083289146423, 1.063184380531311, -0.12171534448862076, -0.12251103669404984, 0.3516448140144348, 0.34926339983940125, 1.3567384481430054, 0.11317235231399536, -0.08742453902959824, -1.4742720127105713, -0.2112540304660797, 0.323416143655777, -1.053284764289856, 0.6940041184425354, -0.07355640083551407, 1.3336830139160156, 0.7079418301582336, 0.237104594707489, 0.6118724942207336, 1.2485958337783813, -1.2536817789077759, -0.8236647844314575, -1.2554876804351807, 1.745711088180542, 0.2614016532897949, 0.0287491288036108, 1.4465289115905762, -1.2046605348587036, -1.308484673500061, 0.808188259601593, -0.29435083270072937, -1.9336498975753784, -1.4904848337173462, -1.5141582489013672, 0.4202558100223541, 0.6157504320144653, -1.331956148147583, -1.2684800624847412, 0.5253084897994995, 0.0008217949653044343, 1.2713557481765747, -0.31072089076042175, 1.1344175338745117, 0.5040098428726196, -0.5482261180877686, -0.5004667639732361, -0.7856583595275879, 1.5343687534332275, -1.4889590740203857, -0.31939008831977844, -0.6622889637947083, 0.4455183446407318, -0.2898995578289032, 0.4087897539138794, -1.6206445693969727, 1.0100151300430298, 0.19767802953720093, 0.5717296600341797, -1.47907555103302, -1.2267975807189941, -0.22254633903503418, -1.586142897605896, 0.4668085277080536, -0.21717391908168793, -1.0597376823425293, 1.0385119915008545, -0.5605014562606812, -0.12737679481506348, 1.8576998710632324, -0.4084589183330536, -0.1295643150806427, 0.29625043272972107, 0.2919260263442993, -0.9374659657478333, 0.1496468484401703, 0.35685569047927856, -0.215385302901268, 0.6199400424957275, 0.30280157923698425, -0.34162160754203796, -0.031664736568927765, -0.4756161570549011, -0.4763234555721283, -2.168671131134033, -0.463687926530838, 0.2686050534248352, 0.5687534809112549, 1.0218151807785034, 1.328322410583496, 1.3252369165420532, -0.9272998571395874, 0.6128857135772705, 0.5415218472480774, -0.6390659809112549, 1.0953564643859863, -0.17927010357379913, -0.6550071239471436, 1.3272017240524292, 1.063166856765747, 0.16204749047756195, -0.05618200823664665, -0.7772531509399414, 0.3314245045185089, -0.6598860621452332, -0.31516313552856445, -0.5300421118736267, 0.003861743025481701, 0.9598275423049927, -0.7029533982276917, -0.303316593170166, -1.429055094718933, -0.7713509202003479, 0.6067787408828735, -1.3288601636886597, -0.4802636206150055, 0.37358036637306213, 0.5725157260894775, -0.6571452617645264, 1.070831060409546, 1.0376369953155518, -0.77293461561203, 0.9284233450889587, -0.2005961835384369, -0.921471357345581, -0.6953652501106262, -2.1979105472564697, 0.06101427972316742, 0.6539336442947388, 0.49056681990623474, 0.6998851299285889, 1.7033277750015259, -0.5913099646568298, 1.6866503953933716, 1.4972821474075317, -0.6053724884986877, 1.5796393156051636, -0.5421828627586365, -0.3435025215148926, 0.5317158102989197, -0.7374951839447021, -0.5683333873748779, -1.0923166275024414, -0.08483204990625381, 0.3577568531036377, 1.101210117340088, 0.05609884858131409, 0.6201116442680359, 1.152835726737976, -0.8362817168235779, 0.2798997163772583, 0.745367169380188, 1.022929072380066, 0.232717365026474, 0.314056932926178, -0.15077629685401917, -1.2809704542160034, -0.8740544319152832, -0.8606612086296082, 1.6039210557937622, -1.2794382572174072, -1.2391400337219238, -0.4428991675376892, 2.0224556922912598, -0.955260157585144, -0.23208683729171753, 1.011421799659729, -0.4728351831436157, -0.4702662527561188, -1.510391354560852, -1.340224027633667, -0.3214113414287567, -0.42669594287872314, -0.0011196312261745334, 0.44446101784706116, 2.0891802310943604, 1.789625644683838, -1.4281553030014038, 0.3973098695278168, -0.516261637210846, -1.6052457094192505, -0.9103185534477234, 1.0105847120285034, -0.1707509607076645, 1.2069157361984253, 0.07460783421993256, 1.9301557540893555, 1.916379690170288, 0.5688303709030151, 1.0148066282272339, -1.7252695560455322, 0.3026276230812073, -1.7161215543746948, -1.608880877494812, -0.220541849732399, -2.0206048488616943, 0.3996362090110779, -0.30975985527038574, -0.07668254524469376, 0.5979151129722595, -0.9005215764045715, 0.029448486864566803, 0.31917837262153625, -0.7358892560005188, 0.6646000742912292, 1.9612315893173218, -0.45865777134895325, 0.8229085803031921, -1.1211365461349487, 1.5319803953170776, 1.8695063591003418, 1.3636515140533447, 0.48279523849487305, 0.5382181406021118, -0.21394333243370056, 1.3964359760284424, 1.8381364345550537, -0.19765906035900116, 0.9688326716423035, 0.19361461699008942, 0.048990726470947266, 1.5994805097579956, 0.09728558361530304, 0.11512839049100876, 0.12832097709178925, 1.3524093627929688, -0.456966370344162, -0.44707226753234863, 0.42409348487854004, -0.30061402916908264, -0.675574004650116, 1.1381866931915283, 0.4028320908546448, -0.9537652134895325, 0.6997829675674438, -0.08105664700269699, 2.158440351486206, 0.028486382216215134, -1.3418748378753662, -1.963689923286438, 0.4064067006111145, 0.5322309732437134, 0.6304357647895813, 0.6997146606445312, -1.6576322317123413, -0.5060970783233643, 0.6699327230453491, -0.08318314701318741, -0.11587461829185486, 2.6376805305480957, 0.4812254309654236, -1.4611518383026123, -0.4121609330177307, -0.6714199185371399, -0.897275447845459, 1.2888576984405518, 1.8882498741149902, 0.3269457221031189, 0.08753813803195953, 0.22359852492809296, 0.36045780777931213, -1.2880247831344604, 0.5350010395050049, 2.4492642879486084, -1.7533756494522095, 0.23103465139865875, 0.7714937925338745, -0.6568766236305237, 0.5148451924324036, -1.4304933547973633, -0.5797138810157776, 0.19308187067508698, 0.25789710879325867, -1.0825226306915283, 1.0604727268218994, 0.16037385165691376, 0.2553933560848236, -0.9184370040893555, -0.9700081944465637, 0.9767375588417053, -1.2397905588150024, 1.8497304916381836, -1.131527066230774, 0.5410638451576233, -0.9426552653312683, -1.5316674709320068, 1.3602601289749146, -0.3246011435985565, -0.6948670744895935, -0.8384499549865723, -1.1737715005874634, 0.19721320271492004, 0.6252787709236145, -0.7619431018829346, -1.2585607767105103, -0.033428750932216644, 0.9903681874275208, -1.782619595527649, -0.3060391843318939, -0.5354143977165222, -0.05375692620873451, 1.30239999294281, -1.0892307758331299, 1.9311108589172363, -0.3349398374557495, 1.8819791078567505, -0.6867603063583374, -1.3735058307647705, -0.7293863892555237, 0.24769175052642822, -0.9840407967567444, -0.27531641721725464, 0.4784882068634033, 0.2032240778207779, 0.6484370231628418, 2.4861667156219482, 0.5777904987335205, -0.5439300537109375, 1.6121561527252197, -0.5202778577804565, -0.0890360102057457, 0.8140116333961487, -0.2778298258781433, 0.7384012937545776, -0.7847464084625244, 1.392773151397705, -0.8440929651260376, -2.567897081375122, 0.0758659839630127, -1.7531514167785645, -1.115654706954956, -1.0477558374404907, -0.7870054244995117, 0.8574396967887878, 1.1506719589233398, 0.9268969297409058, -1.0180943012237549, -0.0782824233174324, 0.5823056101799011, 2.1796767711639404, -1.3738709688186646, -1.9319605827331543, 1.0068366527557373, -0.8088424205780029, 1.9931864738464355, 1.0900228023529053, 0.5918678641319275, 0.6783580780029297, -0.43107062578201294, 0.9291171431541443, 0.5775778293609619, -0.8256000876426697, -0.03998464345932007, -0.0035948262084275484, 0.3215385973453522, 0.16230051219463348, 0.5854762196540833, -0.31756430864334106, -0.2868514955043793, -1.5959054231643677, 1.9888960123062134, -0.5556172132492065, -3.1540451049804688, -1.4899917840957642, 0.22314493358135223, -2.192310333251953, -0.6715078949928284, -0.6267629265785217, -0.27092114090919495, -0.4169660210609436, -0.4564145505428314, 0.899368166923523, -0.1208982840180397, -1.4761868715286255, 0.5616840720176697, -0.3624356985092163, 0.4881775677204132, 0.16718095541000366, 1.5579912662506104, -0.20548665523529053, 0.05471642315387726, -0.3038763701915741, 1.6076503992080688, -0.7754141092300415, 2.597977638244629, -0.13663803040981293, -0.4617742896080017, 1.4566552639007568, -0.7033255100250244, 0.0868895873427391, 0.06429991126060486, 0.5892320871353149, -0.30507078766822815, 1.467911958694458, 0.6749693155288696, -1.1278005838394165, -0.6187493801116943, -0.6599739193916321, 4.4268670082092285, -0.24285928905010223, 0.734472930431366, -0.0002810878795571625, -0.4662989377975464, -0.4094026982784271, -2.0870299339294434, -0.6685945987701416, -1.8387324810028076, -0.7362948656082153, 0.20937354862689972, -0.27274149656295776, 2.272993803024292, 0.24883051216602325, 0.6798974275588989, -0.742836058139801, -0.2384805828332901, 1.3058888912200928, 0.43371689319610596, -0.08599114418029785, 1.6862010955810547, 1.3806536197662354, 2.096059799194336, -0.9321362972259521, -1.5495306253433228, 1.1576449871063232, 0.5589749813079834, 1.2998563051223755, -0.4159243404865265, 0.8798859119415283, 0.7089674472808838, -0.3517175614833832, -0.1447654664516449, -0.701782763004303, 2.106121778488159, 0.5590290427207947, -1.427914023399353, -0.014116144739091396, 0.2941698729991913, 0.8809324502944946, -0.9961299896240234, 1.5956419706344604, -0.9396651387214661, -0.290877103805542, 0.6466952562332153, 0.3978407680988312, -1.0144883394241333, 1.0264837741851807, 1.7414613962173462, 0.1613004505634308, 0.4516713619232178, 0.804715633392334, 0.265959769487381, -1.813471794128418, 2.303901195526123, -0.22421438992023468, 1.6268560886383057, -0.975907564163208, -2.2163586616516113, 1.7939196825027466, -0.09402096271514893, 1.16414475440979, -0.6801249384880066, -0.33390045166015625, -1.2189048528671265, -0.04561534896492958, 0.5727085471153259, -2.0651326179504395, 1.8734560012817383, 0.28953853249549866, 1.92149817943573, 1.099867343902588, -2.4427804946899414, 0.3276560604572296, -0.7867625951766968, -0.11618085205554962, 0.37519726157188416, -0.8424111604690552, -1.6235071420669556, -1.3436224460601807, -0.9008921384811401, -1.7276971340179443, 1.2343213558197021, -0.9724774956703186, 0.8488022089004517, -1.9447810649871826, 0.12044969201087952, 0.8539575338363647, 1.4533988237380981, 1.148913860321045, -0.41038778424263, 0.2138187736272812, -1.1531376838684082, -0.533582329750061, -2.0149292945861816, 1.2738747596740723, -1.5068804025650024, -1.2972500324249268, 1.4753481149673462, -1.6398866176605225, 1.2269611358642578, -1.0425498485565186, 2.649585485458374, 1.7266820669174194, -1.1916518211364746, -1.1340874433517456, -0.43409404158592224, -0.8736684918403625, -0.4929468631744385, 0.791563093662262, -0.9670215249061584, -0.7608420848846436, -0.4801865220069885, -0.14122076332569122, -1.5789504051208496, -1.4708918333053589, 1.2245726585388184, -0.4033070206642151, -1.8789072036743164, 1.2105982303619385, 0.12891046702861786, -1.1917295455932617, -0.6012851595878601, 1.8090161085128784, -0.3468809127807617, -0.32951611280441284, 0.42017316818237305, 0.2648821473121643, 0.33789676427841187, 1.0898106098175049, 0.9237011075019836, -1.6748065948486328, 1.0807838439941406, -1.1674742698669434, -1.1892423629760742, -0.4508821368217468, -0.39054274559020996, 0.37622177600860596, 1.728258490562439, -1.599402904510498, 2.0061986446380615, -0.12906715273857117, 0.6219669580459595, 0.06869897246360779, 1.471503496170044, 0.38760441541671753, -0.30591097474098206, -1.7802804708480835, 0.05968305841088295, 0.115814708173275, 0.9017472863197327, 0.09572704881429672, 0.8026221394538879, -0.3985719084739685, -0.1760978102684021, -1.291627287864685, 1.1493901014328003, 0.8952233791351318, 0.40456506609916687, 1.2029424905776978, 0.5575355887413025, -0.9090420603752136, 0.3602554500102997, -0.429993599653244, 0.38663211464881897, -1.6943509578704834, -1.704053521156311, -0.2003789246082306, 1.02887761592865, 1.0531867742538452, 0.5018714666366577, 0.2831704914569855, -1.000693678855896, 0.5822255611419678, 0.5813480615615845, -0.2689981460571289, -0.3102418780326843, 1.1382426023483276, 0.08028662204742432, -1.0589932203292847, 1.1855523586273193, 0.10208209604024887, -0.8728966116905212, -0.8656323552131653, 0.6394837498664856, 1.7916065454483032, -0.2714531719684601, -0.009968572296202183, -0.912614107131958, -2.2316949367523193, 0.2279369980096817, -0.322340190410614, -0.29121169447898865, 0.46873119473457336, 0.049390699714422226, 1.8386374711990356, 1.04231858253479, -0.21402017772197723, -1.6538952589035034, -0.17832447588443756, -1.9460614919662476, 1.563454031944275, 0.43991294503211975, -1.5870555639266968, -0.648144006729126, -0.8238874673843384, 0.4884958863258362, -1.4384410381317139, -1.0986723899841309, -0.4074363112449646, 1.2258332967758179, 0.7666433453559875, 0.026510702446103096, -0.5984803438186646, 1.608613133430481, -0.013538929633796215, 0.6361578702926636, 1.163834571838379, 0.2082141935825348, -0.4948117434978485, -0.9619673490524292, -1.740539312362671, 1.7189621925354004, -0.8386282920837402, 1.0953360795974731, -0.643807590007782, -0.2690213620662689, -0.881177544593811, 0.7700096368789673, 0.715019941329956, -1.0850588083267212, -1.372376561164856, 0.7921918034553528, -0.580647885799408, 0.8757383823394775, 1.8859330415725708, -0.17493489384651184, -2.1677284240722656, 0.6578895449638367, -1.329248309135437, -0.7936223745346069, -0.14728178083896637, 0.8248591423034668, 1.5965428352355957, -1.9043173789978027, 0.9449567198753357, 0.9830911755561829, 0.4819863736629486, -0.37515518069267273, 1.4627861976623535, -0.42305269837379456, -1.321407437324524, -0.9176559448242188, -0.3350647985935211, 3.341510772705078, 0.2075381577014923, -0.37025922536849976, -0.7167661786079407, -1.3611595630645752, -0.028639480471611023, 0.22654177248477936, 1.2774897813796997, 0.7235331535339355, -1.1756633520126343, -0.7345795631408691, -2.225172758102417, -0.8223649263381958, 0.22540712356567383, -1.0566405057907104, -1.9691263437271118, -0.9780840873718262, 1.1563550233840942, -0.17769059538841248, 0.08406349271535873, -0.569602370262146, 0.1975516974925995, 0.06610969454050064, -0.7509399652481079, 1.71562659740448, 0.8174954056739807, -0.4019888639450073, -0.12103520333766937, 2.0787742137908936, 0.275069922208786, -1.1916319131851196, 0.803043782711029, 0.8980370163917542, 1.0482416152954102, 1.6355313062667847, 1.1668920516967773, -0.9522210955619812, -0.48619943857192993, -0.006829020567238331, -0.03242570906877518, -1.5859735012054443, 1.0004217624664307, -0.6792222261428833, 0.41420722007751465, -0.7640576362609863, -0.6799193620681763, -0.6392715573310852, 0.43234410881996155, -0.04971685633063316, 0.19954273104667664, 0.4018568992614746, 0.3225575089454651, 0.15985354781150818, -0.316813588142395, -0.32141196727752686, 1.1220285892486572, 1.1986440420150757, 0.31286877393722534, 1.228895902633667, -0.5547020435333252, 0.3270834982395172, -1.671081304550171, 0.006364470347762108, -0.3603130877017975, -0.011973547749221325, 1.0053222179412842, -1.1244394779205322, -1.009837031364441, -1.2557086944580078, -1.4494386911392212, 1.353049874305725, -0.23147402703762054, 1.1973633766174316, -0.032427091151475906, -0.24752813577651978, 0.29804500937461853, -0.2692252993583679, -0.6397801041603088, 1.0170700550079346, 0.32128968834877014, -0.8165012001991272, 0.9560646414756775, -0.1865883767604828, -0.2294503152370453, -1.2269742488861084, 1.6016590595245361, -0.9107540249824524, 0.5945355296134949, 0.5203179717063904, 0.06679314374923706, -0.6012205481529236, -0.04476935788989067, -0.6516098380088806, 0.4719602167606354, -0.02032441645860672, -0.9620645046234131, -3.4893577098846436, -0.7088544368743896, -1.0314823389053345, -2.7298367023468018, -0.3249491751194, 1.3849962949752808, -1.247125267982483, -0.8581575751304626, -0.42957592010498047, 1.2200783491134644, -2.0167758464813232, 0.1240905374288559, -0.8711252212524414, 0.6318626999855042, 0.8972691893577576, -0.034196920692920685, -0.24328573048114777, 1.3943639993667603, 0.3715324401855469, -0.705532968044281, -0.7512911558151245, 1.4824280738830566, -0.5648486018180847, 0.6670267581939697, -0.39253687858581543, 0.7500092387199402, -2.2856740951538086, 0.9752926230430603, -0.44464632868766785, 0.0374329648911953, 1.0794222354888916, 0.4936457574367523, 1.4806432723999023, 0.09441987425088882, 1.0479968786239624, 0.6977023482322693, -0.3053876459598541, -0.19099576771259308, 0.2190595269203186, -0.2484573870897293, 2.0880651473999023, 1.7183679342269897, -0.3878147304058075, -0.9256052374839783, -1.2525979280471802, -0.44472774863243103, 0.26189562678337097, -0.1218794584274292, 1.6735867261886597, -1.5561922788619995, 0.740705668926239, -0.9827929735183716, -2.0889227390289307, -2.403385639190674, 0.5965694189071655, 0.19536332786083221, -0.592832624912262, -0.31305232644081116, 0.0855337455868721, -0.17906470596790314, -0.46581533551216125, 0.3662530183792114, 0.22444087266921997, -1.9224642515182495, 1.3354003429412842, -1.1526422500610352, 2.8753163814544678, -1.7686209678649902, -0.9425956010818481, -1.9928711652755737, 0.5442224740982056, 1.304443359375, -0.42119747400283813, -0.20165735483169556, -0.4468308091163635, 0.03766927868127823, 1.2907480001449585, -0.32420462369918823, -0.7281444072723389, -0.17696313560009003, 0.2031300663948059, -2.5571908950805664, -1.0162320137023926, 0.20020125806331635, 0.1434621959924698, 0.45478329062461853, -1.3634827136993408, -0.40479353070259094, 1.2875608205795288, 1.9638913869857788, 1.3038172721862793, -1.3533291816711426, 1.1669442653656006, -0.887035071849823], [1.4866557121276855, 1.0379328727722168, 2.1124424934387207, -0.46899959444999695, -0.23921842873096466, 0.2839028239250183, 0.3651579022407532, -0.13049422204494476, 0.945185661315918, 3.2721025943756104, -0.1657099723815918, 1.6110310554504395, -0.521097719669342, 0.5153080224990845, -0.10775800049304962, -0.5301974415779114, 0.7535392642021179, -0.4440171718597412, -0.108862966299057, 0.8625277876853943, 0.6375055313110352, 0.9846177697181702, -1.101356863975525, -1.0879427194595337, 1.256316900253296, 1.463632345199585, 1.038577914237976, 0.7546431422233582, 1.299702525138855, 1.13310968875885, -0.10717654228210449, -0.10632415860891342, -0.8499714136123657, -0.9963789582252502, -0.11052106320858002, -0.677310585975647, 2.124735116958618, -0.8194389939308167, -0.9756810069084167, 0.34510406851768494, 1.4405597448349, -0.37141087651252747, 0.3728336691856384, -1.472688913345337, -0.6779634356498718, 0.5324071645736694, 0.31883659958839417, -1.6964114904403687, -0.12289644032716751, 1.4229868650436401, 0.3004821836948395, -0.1423638015985489, -0.5930430293083191, -0.5226214528083801, -1.6010600328445435, 0.45908981561660767, -0.20910532772541046, 0.2516314685344696, -2.366863965988159, 0.807736337184906, 0.7324383854866028, 1.392723560333252, 1.4135814905166626, -0.1188177615404129, 0.5751150250434875, -0.6315034627914429, -0.7929869890213013, -0.8540719151496887, 0.5287573337554932, -0.9621342420578003, -1.578822374343872, -0.3625807464122772, 0.21856072545051575, 1.5445520877838135, -0.8119944334030151, -1.2296210527420044, 0.594609797000885, -0.471174955368042, -0.8576021790504456, 0.08333060890436172, -0.4071686863899231, -0.9152203798294067, -0.45295780897140503, 0.3849683105945587, 0.14196746051311493, -2.0933802127838135, 1.8129514455795288, -0.5989869832992554, 0.43085405230522156, -0.153229221701622, 0.32497283816337585, 0.09628573060035706, -0.3696550130844116, 0.5308467149734497, -1.3991178274154663, -0.8426539301872253, 0.0007205063011497259, -0.2736041843891144, 0.2241261750459671, 0.09506332874298096, -0.3101963698863983, 0.06138717383146286, -0.45280084013938904, 0.021444348618388176, -1.2555277347564697, 0.007497734390199184, -1.7014684677124023, 0.8964661359786987, 0.4140763580799103, 0.27830764651298523, -2.396692991256714, -0.9873352646827698, -0.17750109732151031, 0.990418016910553, -1.691162347793579, -0.1807653307914734, 2.0810203552246094, 0.9883814454078674, -0.8619170188903809, -0.9623885750770569, -0.21575899422168732, -0.004957976285368204, -0.08312209695577621, -0.42477262020111084, -1.3584891557693481, -0.09309927374124527, 1.1782302856445312, 1.1898770332336426, -0.9127790927886963, 1.170029878616333, 0.2599834203720093, -0.5101022124290466, -0.5464110374450684, 0.2651048004627228, -1.9131051301956177, -0.10881231725215912, -0.07681199163198471, 0.5192440748214722, 2.1176235675811768, -0.48537373542785645, 0.7857850790023804, 0.577877402305603, 0.0521915927529335, 0.05106159672141075, -0.7404935359954834, 0.6183752417564392, 1.175508737564087, -0.5479367971420288, 1.22525155544281, -0.6095026731491089, 1.1537185907363892, -0.5386265516281128, -0.9228533506393433, -0.2377137541770935, -2.2066097259521484, -0.9582790732383728, -0.17123529314994812, 1.6773416996002197, -0.41267579793930054, 0.10932285338640213, -0.10865136981010437, 0.5506859421730042, 0.5090299844741821, -0.09802478551864624, -0.994577169418335, -0.30742061138153076, -1.3850555419921875, 1.0958178043365479, -0.42902782559394836, -1.8753020763397217, 0.016692252829670906, -0.7219749689102173, -0.8817697167396545, 0.0011470600729808211, 1.2213828563690186, 1.0403469800949097, -0.651912271976471, 0.6871891617774963, 1.0496087074279785, 2.768298864364624, 0.37244850397109985, 1.285394549369812, 1.650329351425171, 0.10109086334705353, -0.3782941699028015, -0.4032057225704193, -1.3557450771331787, 1.7127203941345215, -0.7731476426124573, -0.7160776257514954, -0.1786872297525406, -0.8115277290344238, 0.3448009192943573, -0.556659996509552, -0.1560194194316864, -0.3909907937049866, -2.4202120304107666, 1.2547842264175415, 0.7250745892524719, -1.7616530656814575, -0.20234350860118866, -1.2009427547454834, 0.23957107961177826, 0.09369887411594391, 2.0065274238586426, -0.7728683948516846, 0.24441422522068024, 0.4801322817802429, 1.0936990976333618, -0.32081282138824463, 1.0410058498382568, 0.006816553417593241, 0.020993974059820175, -0.6435462832450867, -1.0344150066375732, -1.4301424026489258, -0.4565708041191101, 0.3097500503063202, -0.8687955737113953, 0.8464841842651367, -1.067553997039795, -1.1737929582595825, -1.1852211952209473, 1.3969416618347168, -0.2583872377872467, -0.47659793496131897, -0.9312552809715271, -0.07274578511714935, 1.8665540218353271, 0.7276684045791626, -0.08752183616161346, 1.2884029150009155, 0.13162679970264435, -0.2118532806634903, 0.6379678249359131, -0.4085099697113037, 0.2799379825592041, 0.08152294903993607, -0.6017820835113525, -0.5589614510536194, 2.1518614292144775, 0.3671448826789856, -0.1264471709728241, -0.4367885887622833, 1.2728804349899292, 1.4672839641571045, -1.0389760732650757, -0.008973637595772743, -1.1605141162872314, 0.5458252429962158, 0.25516438484191895, 0.1834297776222229, 0.9449425935745239, -0.5372147560119629, 0.428398460149765, 0.4688863754272461, 0.7393653988838196, 0.10641223937273026, -0.538449227809906, 1.210524559020996, 0.052349548786878586, -1.0414113998413086, -0.7234278917312622, -0.6105483770370483, 1.092653512954712, -0.6055504083633423, -1.1934196949005127, 0.3101655840873718, 0.39816367626190186, 0.5320724844932556, 0.8127914071083069, -1.894679069519043, -0.28953254222869873, 0.5424873232841492, -0.3951217532157898, -0.5411851406097412, -1.737945318222046, 0.30737730860710144, 0.36415329575538635, 0.27877628803253174, 0.5329331159591675, 0.03543565049767494, 0.6135238409042358, 0.9276438355445862, 3.4162402153015137, -1.2302149534225464, -0.08357933908700943, 0.10451943427324295, 1.5294902324676514, -1.4084736108779907, 1.3473670482635498, -0.9829012155532837, 0.09453186392784119, -0.697131872177124, 0.3716888725757599, 1.8425824642181396, -0.6533554792404175, -1.7080001831054688, 0.1500883549451828, 0.014655888080596924, 0.26743820309638977, -0.1817132532596588, -1.584134578704834, 0.4063487946987152, -0.905527651309967, -0.08122117817401886, 0.25269225239753723, -1.4283206462860107, 0.16233684122562408, -0.9787061810493469, 0.5705695152282715, 0.5864655375480652, -0.9721769094467163, -1.1382193565368652, 0.33963704109191895, 0.8361884951591492, -1.851939082145691, -1.43918776512146, 1.031874179840088, -2.150036096572876, 0.09897447377443314, 0.6774415373802185, 1.176755666732788, 0.5951219201087952, -0.028866969048976898, 0.15020963549613953, -1.7332850694656372, -0.23805037140846252, 1.0854253768920898, -1.8348894119262695, -1.5836938619613647, -0.5331021547317505, -0.6854817271232605, -1.696744680404663, -0.8930829763412476, 3.060911178588867, 1.9823508262634277, 0.8318800926208496, -0.9811117649078369, 1.976265549659729, 1.1131055355072021, 0.3943106234073639, -0.4654516875743866, 1.3249866962432861, 1.0236444473266602, 1.391233205795288, -1.2512611150741577, 0.4175848066806793, 1.2375116348266602, 2.302373170852661, 0.8076770901679993, 0.9966199994087219, -0.900039792060852, 0.10697674751281738, -0.30662891268730164, -0.3502459228038788, -1.1109129190444946, -1.506909728050232, 0.5585538744926453, -2.084866523742676, 0.5212225317955017, 0.23829534649848938, -1.0146769285202026, -0.35305991768836975, 0.9590643048286438, 0.13460949063301086, 0.1570870578289032, -1.8889436721801758, 0.29806792736053467, 0.8149241209030151, -0.10485893487930298, 0.5021160840988159, 1.2268093824386597, -1.2177047729492188, 0.27242711186408997, -0.30095481872558594, 0.5057341456413269, 1.4984077215194702, -0.8628957867622375, -0.8352221250534058, -0.40063631534576416, 0.34417587518692017, -0.5555410981178284, 0.9461051225662231, -0.8357464075088501, -0.7769690752029419, -0.10482030361890793, 0.2418418824672699, 0.410500705242157, 1.1166212558746338, 0.17366352677345276, 0.944791853427887, -0.4878154695034027, 0.1620939075946808, -0.5791420340538025, -0.6417765021324158, -0.11071427166461945, 0.6392471790313721, 0.21478047966957092, 1.100204586982727, -1.06708824634552, -1.6243010759353638, -0.017414242029190063, 0.9155958294868469, 0.18623484671115875, -1.2648725509643555, 0.1904676854610443, 1.2816083431243896, -0.9362777471542358, -0.020319903269410133, 0.1993819773197174, 0.16594050824642181, -0.0761943906545639, 0.12531761825084686, 0.30732667446136475, 1.195377230644226, -0.00954800471663475, -0.6976369619369507, -0.22914043068885803, 1.5198545455932617, -0.4882809519767761, -0.2874368727207184, 1.0546889305114746, 1.5327229499816895, 1.698996663093567, 0.2578293979167938, -2.10561466217041, -1.3598097562789917, 0.6127245426177979, -1.6765247583389282, 0.10306112468242645, 0.1804935485124588, 0.04712147265672684, -0.5052143335342407, -2.245793581008911, 0.9814490079879761, 0.25979021191596985, -1.6007128953933716, -2.1632256507873535, -1.2548537254333496, -0.7337952256202698, 0.019624486565589905, -0.8898836970329285, 0.8431618809700012, -0.20778383314609528, 0.900100827217102, -0.422818124294281, -1.5294615030288696, -0.7879020571708679, -0.3093910813331604, 1.246706485748291, -0.44572606682777405, -1.8689868450164795, 0.5367007851600647, -1.0294458866119385, -1.1602983474731445, -0.5137764811515808, 0.899109423160553, 0.29443731904029846, 0.1676325649023056, 0.9771191477775574, -0.36750468611717224, -1.372280478477478, -0.21459488570690155, 0.382994681596756, -0.7208213210105896, -1.711354374885559, -0.8725477457046509, 0.4898933470249176, -0.5349632501602173, -1.6714750528335571, -0.4537809193134308, -1.330807089805603, -2.37040638923645, 1.6145939826965332, -1.7462139129638672, -1.1698856353759766, -1.875473141670227, -0.35160988569259644, 0.6017550826072693, -0.4566047191619873, -0.49271777272224426, 0.05503731220960617, 0.13975919783115387, 0.6615332961082458, -0.3231353163719177, 1.5316001176834106, -1.0004522800445557, -0.5060951709747314, 0.9515697956085205, -0.029861733317375183, 1.9740716218948364, -0.03866298124194145, 0.24673470854759216, -0.1641581505537033, -0.4925355315208435, 0.6916411519050598, -0.9422928094863892, -0.39229631423950195, -1.2641040086746216, -1.9839885234832764, 0.2027047574520111, -0.7575873136520386, -1.0271220207214355, -1.139570713043213, -0.2524093687534332, -0.2443157136440277, -0.19178447127342224, -2.415249824523926, 0.5960636138916016, -0.8636712431907654, -0.7313119173049927, -0.3766304552555084, -0.9299309849739075, -0.355764776468277, 0.7495434880256653, -0.8170395493507385, 1.0608937740325928, 0.6787790060043335, 0.3678942918777466, -0.3057297468185425, 0.9663325548171997, -0.25936606526374817, -0.9655027985572815, 0.0630829781293869, -0.005433667451143265, -1.9621235132217407, 0.48842689394950867, -0.029894011095166206, 0.3568202257156372, 1.1418746709823608, 1.2753126621246338, 0.3935263156890869, 0.8722391724586487, -0.47597649693489075, 0.7662227153778076, 0.007759292609989643, -0.20392540097236633, 1.1490232944488525, 1.4912052154541016, 0.4100196361541748, -0.28454333543777466, 0.37534433603286743, 0.48970264196395874, 0.8187269568443298, 1.5785006284713745, -0.11050400137901306, 1.6693274974822998, -0.8130419850349426, 0.3240770399570465, 0.002830279991030693, -1.00577712059021, 2.066511631011963, -0.5577820539474487, -0.3872349262237549, -1.293533444404602, 1.834596872329712, -0.13705217838287354, 0.9281471967697144, 0.27353939414024353, -0.8629994988441467, 1.8618690967559814, 0.5472065806388855, -0.46193641424179077, -0.3214096426963806, -0.4072476625442505, -1.1202179193496704, -1.0973697900772095, -1.1947364807128906, 0.7272217273712158, 0.05640744790434837, -0.008210777305066586, 0.22934426367282867, 0.5428379774093628, 0.5241539478302002, -0.7435516715049744, -0.477761834859848, 0.91059809923172, -0.5130570530891418, -0.29465770721435547, -1.648676872253418, -1.4414527416229248, -1.1114052534103394, -0.23128974437713623, 0.02468234673142433, -0.18921153247356415, -0.9639357924461365, 1.355455994606018, -0.49891534447669983, 1.1947822570800781, -0.42164158821105957, -0.6140408515930176, -0.3662380278110504, -0.03332856670022011, -0.8008341193199158, -2.935462713241577, -1.5300673246383667, -2.3155064582824707, 1.574150800704956, -0.057021815329790115, -0.543890655040741, -0.10500999540090561, 0.2854912281036377, 0.6318944096565247, 1.2372454404830933, -0.9427886009216309, 0.2992721498012543, 0.261653870344162, -1.1854647397994995, 0.6855462193489075, -0.22915443778038025, -1.699183702468872, 0.6596068739891052, 0.3156292140483856, 0.161721333861351, -2.0207386016845703, -0.6552667021751404, -0.4187382757663727, 0.14471526443958282, 0.018929358571767807, -0.9004480838775635, 2.8976049423217773, 0.04057731851935387, -0.6877662539482117, -0.4830089509487152, 0.35133907198905945, 0.628157913684845, 0.30541059374809265, -1.1575806140899658, -0.45594751834869385, 1.1548800468444824, -1.403749704360962, -0.014058370143175125, 1.1693850755691528, 0.5647851824760437, -0.822482168674469, -0.9001561403274536, 0.8642878532409668, 0.7569631934165955, -1.1495764255523682, 1.1910817623138428, -0.7851264476776123, -0.10530311614274979, -0.215693861246109, 0.2895641624927521, -1.1456607580184937, -0.7853391766548157, -0.4109884798526764, -0.9319574236869812, 1.5961644649505615, 0.13256041705608368, -0.40829402208328247, 0.6032207608222961, -0.989701509475708, 0.8664460778236389, 0.7205817699432373, -1.0841304063796997, 0.2763364613056183, -1.5394203662872314, -0.5398310422897339, -1.5679632425308228, 1.9175958633422852, -1.9619051218032837, -2.235469341278076, 1.7209961414337158, 0.20072093605995178, 0.8978206515312195, -0.7607285380363464, -0.3997924327850342, -0.11243679374456406, 1.451469898223877, -0.7087135910987854, 1.5341533422470093, -2.240001678466797, -1.3508367538452148, 0.12330745905637741, 0.2870495021343231, 0.8348459601402283, 0.6065581440925598, 0.8072259426116943, -1.4591301679611206, -1.8688288927078247, 0.4692457318305969, 0.6636539101600647, 0.01839526742696762, -0.010398915968835354, 1.45163094997406, 1.0958439111709595, -1.7765434980392456, 0.7041708827018738, -1.6488440036773682, -0.5098952651023865, -0.6660956144332886, 0.7543308734893799, -0.008958584628999233, -3.2146551609039307, -1.4809647798538208, -0.15693485736846924, -0.42295825481414795, 1.031868815422058, 1.180903434753418, 0.12602347135543823, -0.13228802382946014, -0.8809496760368347, 0.43053221702575684, -1.1857022047042847, 1.5313286781311035, 1.0547723770141602, 1.6091828346252441, -0.8221120238304138, 1.645140290260315, 0.2791309654712677, 0.3504367470741272, -0.32204559445381165, 1.692085862159729, -0.733390748500824, 0.4889881908893585, -0.18889307975769043, 1.2155100107192993, 0.15798825025558472, 0.5899849534034729, -1.6782654523849487, -0.8326097130775452, -0.2557426691055298, -1.7404781579971313, 1.0553221702575684, -0.49128809571266174, -0.750701904296875, 1.0756077766418457, 0.7076300382614136, 0.8975537419319153, -0.6738278865814209, 1.0727280378341675, 0.2226780205965042, -0.2878299355506897, 0.5194413065910339, 1.8439087867736816, 0.5002657771110535, -0.3246125280857086, 0.8925450444221497, 1.0579184293746948, 0.9225917458534241, 0.7654895186424255, -3.2818846702575684, -0.8286399245262146, -0.9185815453529358, 0.7830260992050171, 0.0057824719697237015, 1.1561815738677979, -1.439721703529358, -0.1473091095685959, 0.6263453364372253, -0.9178889989852905, 2.2441482543945312, 0.9922623038291931, -0.5258579850196838, 0.25055328011512756, 0.06208697706460953, 0.0860351100564003, 2.2111823558807373, -0.8046970963478088, -0.5091647505760193, -0.4596359431743622, 0.023888899013400078, 0.48643267154693604, 0.12469656020402908, -0.5336123704910278, -0.6699530482292175, -1.28287935256958, -0.2068319171667099, -0.48327797651290894, 1.2067514657974243, -1.7779428958892822, 1.5411882400512695, -0.8112789392471313, 0.46947309374809265, 1.1016970872879028, 0.9867841005325317, 0.13840016722679138, 0.7969996929168701, 0.20005349814891815, 0.5940879583358765, 0.5910565257072449, 0.3185034990310669, 0.7094555497169495, 0.35944610834121704, 0.01632433757185936, 1.3294168710708618, 0.6734713315963745, -1.068781852722168, 1.0585038661956787, 1.0181140899658203, -1.265541911125183, -1.0505448579788208, -1.4124382734298706, -0.5095661878585815, -0.5258480906486511, -1.2416489124298096, 0.2917982041835785, 0.15658338367938995, -0.08678815513849258, -0.7316741943359375, -1.0840078592300415, -0.13752368092536926, -0.7793813943862915, 1.2786836624145508, 0.15972304344177246, 2.8915927410125732, -1.0018246173858643, -0.0761476531624794, 0.523232102394104, -1.7979792356491089, 0.5528526902198792, -0.5503710508346558, -1.3379603624343872, -1.8482273817062378, 0.25627610087394714, 0.3238888382911682, -0.7340072989463806, -0.06977802515029907, -0.9593789577484131, 1.4982839822769165, 1.308915376663208, 0.24421502649784088, 0.7450616955757141, -0.081144317984581, 0.6636022329330444, -0.7182496190071106, -0.1988133043050766, -0.15650135278701782, 0.3754984438419342, -1.0183959007263184, -1.1800786256790161, -0.1903495341539383, 0.22261649370193481, -1.2364267110824585, -1.1624574661254883, 1.2186298370361328, 0.08693865686655045, 1.228053331375122, -0.9369136691093445, -0.6273145079612732, 0.6908247470855713, 0.4585590064525604, -0.9591817855834961, -1.227834939956665, -0.31888318061828613, 0.5561201572418213, -0.26815134286880493, -0.5144572854042053, -0.22160494327545166, 0.29906144738197327, -0.8781772255897522, -0.8687033653259277, -0.30665841698646545, -0.13951455056667328, -0.32654809951782227, -0.4259539246559143, -0.4958086311817169, -2.260636329650879, -0.19634075462818146, -1.1491559743881226, -0.28763270378112793, 1.3346627950668335, -0.5129625797271729, 0.6618778705596924, 1.4099732637405396, -1.2987060546875, 0.009689233265817165, -1.141674280166626, 1.4008156061172485, 0.6593621373176575, -0.25514060258865356, -0.7242993712425232, 1.8924729824066162, -1.01695716381073, -1.5687897205352783, 1.4766958951950073, -0.84722900390625, -0.3149718940258026, 0.4955599009990692, 0.766974925994873, -1.2698352336883545, 1.4655035734176636, 0.6239514350891113, -0.9772418737411499, -0.6932680010795593, -1.878617763519287, 1.3054794073104858, -0.25928983092308044, 1.120308756828308, 0.1101212352514267, 0.7822489142417908, 1.614139437675476, -0.8899233341217041, -0.5708988308906555, -0.056142114102840424, 0.6679772138595581, -0.5945013761520386, -0.4325922429561615, -0.3659341633319855, -0.5369612574577332, 0.8888083696365356, -0.32132768630981445, 0.2865942120552063, 1.4379019737243652, -1.5140429735183716, -1.5708630084991455, 0.5674725770950317, -0.43169382214546204, -0.6856496334075928, 0.26942822337150574, 0.7620114684104919, -0.2972106635570526, 0.7515697479248047, -0.9316406846046448, 0.4172412157058716, -0.8127407431602478, 0.5440760850906372, -0.24645091593265533, 0.6955589652061462, -1.06182062625885, 1.084448218345642, 1.207777976989746, -0.310608446598053, 0.39723512530326843, 1.7702142000198364, 2.0572071075439453, -0.9375385046005249, -0.974953830242157, -0.2314128875732422, -0.24333234131336212, -1.6189095973968506, -0.25142788887023926, 0.7513098120689392, 0.12624889612197876, 0.04376845806837082, 0.987474262714386, 0.5657263994216919, 1.746667504310608, -0.2760763466358185, 1.1126281023025513, 1.038308024406433, -1.4591126441955566, 1.0999542474746704, -1.0295754671096802, 1.4365922212600708, 1.6633906364440918, 1.5799752473831177, 0.6586143970489502, 0.5395077466964722, 1.0491554737091064, -1.5471068620681763, -0.9571017026901245, 0.7800671458244324, -0.4093155264854431, -0.5988340377807617, 0.8183411955833435, -1.1557122468948364, 0.15601131319999695, -0.04260069876909256, -0.9372993111610413, 0.7373822331428528, 0.47112733125686646, -1.8843411207199097, -1.3827948570251465, 0.7903217077255249, 0.49702996015548706, -0.4779157340526581, 3.0318799018859863, 0.018317602574825287, -1.721954584121704, 1.436295509338379, -0.2290872037410736, 1.0345169305801392, -0.4020407199859619, -1.0020418167114258, -0.3036787807941437, -0.6058390736579895, -0.9736103415489197, 0.8604300618171692, -1.3874739408493042, -0.01571541652083397, -0.18125709891319275, -0.35079988837242126, 1.074055790901184, 0.7454238533973694, 0.5920994281768799, 0.07020988315343857, 0.555063009262085, 0.5163376331329346, 1.2073029279708862, 2.0805444717407227, -0.9681339859962463, 0.5420526266098022, 0.19340980052947998, 0.36977460980415344, 0.387961208820343, -0.3625306487083435, -0.8305213451385498, 1.1144031286239624, -0.10414005815982819, 0.47218087315559387, -0.313920795917511, -1.265624761581421, 2.099132776260376, -0.6374763250350952, 1.0984983444213867, -0.41339609026908875, 0.17856255173683167, -0.713054895401001, -0.03502165153622627, 1.284073829650879, -0.1469448059797287, -0.37824690341949463, 1.7782527208328247, -0.6331768035888672, 1.6538299322128296, 0.3957585096359253, 1.363281488418579, 1.249984860420227, 0.35004690289497375, -0.9380161762237549, 0.89445960521698], [0.4448396563529968, 0.4334602355957031, -1.013169765472412, 0.7084373235702515, 0.49089446663856506, 1.9813956022262573, -1.3021513223648071, 0.3166060447692871, 0.9133666753768921, 0.589790940284729, -1.1420425176620483, 0.1787930577993393, -0.7034905552864075, -0.1421285718679428, 1.013033390045166, 0.0078507075086236, 1.4533047676086426, 1.7129122018814087, -0.6649779677391052, -0.7445784211158752, 0.9675303101539612, 0.20306351780891418, -0.3527204990386963, -0.7496896386146545, -0.10655655711889267, 2.103084087371826, 0.013538980856537819, 1.3222787380218506, 2.0475332736968994, -0.7013947367668152, 0.48410698771476746, 0.10102792829275131, -0.12158235162496567, -0.8605917096138, 0.9308058619499207, -1.7761735916137695, -0.803512454032898, 0.24581487476825714, -1.4525967836380005, -0.38687294721603394, 0.12536659836769104, 0.09304914623498917, -0.22931700944900513, 0.4991777837276459, -0.31250181794166565, 0.07006800174713135, 0.3006276488304138, 0.045448072254657745, 0.2700364887714386, 2.0452144145965576, -1.3428246974945068, -1.681314468383789, 0.9277088642120361, -0.24527409672737122, -0.6359347701072693, 1.4893205165863037, 1.309637427330017, -1.8508356809616089, 1.1600828170776367, 0.3488382399082184, -0.03974655270576477, 1.4994964599609375, 0.6708239316940308, 1.0035676956176758, -0.15843603014945984, 0.436981201171875, 0.005621245130896568, -0.7731909155845642, -2.203695297241211, -0.7735518217086792, -0.2836557924747467, -0.5788084268569946, -0.7491669654846191, -0.45534929633140564, 0.5609575510025024, 1.463942289352417, 1.3045092821121216, -0.07775884121656418, -0.40397870540618896, -0.14694441854953766, 0.560330331325531, 0.5685168504714966, -1.1530956029891968, 0.2229604870080948, 0.09819988906383514, -0.4607256352901459, 0.016684889793395996, 2.2931766510009766, -0.4938579499721527, -1.692077398300171, 0.5016767382621765, -0.4807816445827484, 0.578345775604248, 0.3709809482097626, 0.3248271942138672, 0.966381847858429, -1.1117976903915405, -0.20777079463005066, 1.4851042032241821, -1.4493502378463745, -0.5167682766914368, -0.09284769743680954, 0.1631585657596588, 0.40530455112457275, -0.9484351277351379, -0.19246508181095123, -0.8414484262466431, 1.3191391229629517, 1.7693884372711182, -0.12390812486410141, -0.384576678276062, 1.0211015939712524, -2.2767107486724854, 1.901231288909912, 0.7945987582206726, 1.2555962800979614, 0.2542913258075714, -1.0992785692214966, 0.4963376820087433, 0.3889977037906647, -1.0689661502838135, -0.5535687208175659, -1.0660134553909302, -0.1454281210899353, 0.84874427318573, -0.5960025191307068, 1.2473169565200806, -0.9442448616027832, -2.046776294708252, -1.040462613105774, 0.38351279497146606, 0.18811094760894775, -0.9166936874389648, -1.2531344890594482, -1.356691837310791, -1.814470648765564, 1.0889546871185303, 1.7711695432662964, 0.5431243777275085, 1.0269197225570679, -2.2668192386627197, -0.6218233108520508, 0.7970863580703735, 1.0085335969924927, 0.014575029723346233, 0.13902994990348816, 0.8804036974906921, -0.6659265160560608, 1.0994048118591309, 0.5391129851341248, -0.9841749668121338, -0.05995270982384682, -2.224982738494873, 0.5738566517829895, 0.012853859923779964, 0.8668316602706909, 0.8569085597991943, -0.17752571403980255, -0.4123012125492096, 0.036029472947120667, 1.2680081129074097, 0.4610249698162079, -0.231174498796463, 0.6215212941169739, 1.23306405544281, 1.4862982034683228, 1.962194800376892, 0.11304477602243423, 0.9559696316719055, -2.1730704307556152, -0.14747470617294312, 0.6877317428588867, -0.27171790599823, 0.463625431060791, 0.7192434668540955, -0.3437095284461975, 0.0005154820391908288, 0.11313401907682419, -1.503528356552124, -0.20759515464305878, -0.3125487267971039, 0.745211660861969, 0.4716701805591583, -0.9258133769035339, 0.2336820662021637, 0.4230019152164459, 3.0006651878356934, -0.7763771414756775, 1.7357853651046753, 0.8446798920631409, -0.19055277109146118, -0.40242689847946167, -0.2654716372489929, 1.002329707145691, -1.08827805519104, 0.6799894571304321, -0.24290162324905396, 1.018547534942627, -0.5206746459007263, -1.0728453397750854, 1.306230902671814, -0.09599065780639648, -0.8151048421859741, -0.47087544202804565, -0.7956074476242065, -0.8698087930679321, 0.03959551453590393, -1.1433724164962769, -1.5102745294570923, 0.6115564703941345, -0.0786452516913414, -0.2516869008541107, -0.9546800255775452, -0.7276167869567871, 0.31480494141578674, 0.32264500856399536, 0.350269079208374, 0.41015493869781494, 0.8327239751815796, 0.9646493792533875, 0.412638396024704, -0.3248157203197479, -1.1427123546600342, 0.6185929179191589, 0.3490096926689148, -0.9736925959587097, -1.9898089170455933, 0.8994835615158081, 0.28417032957077026, -0.8243921995162964, 0.760359525680542, -2.021564483642578, -0.2652919590473175, -0.12920822203159332, -1.3123023509979248, -0.7972727417945862, 0.5190519094467163, 0.6181092858314514, 1.1373026371002197, 0.5011895298957825, 1.7232600450515747, 0.5295128226280212, 1.330127477645874, -0.09311053901910782, 0.2918943762779236, -0.16974087059497833, 0.3168443739414215, 1.4817585945129395, -1.549784541130066, 1.131614327430725, -1.0586072206497192, -0.9055196046829224, -1.9065583944320679, 0.45192933082580566, 0.23396077752113342, -1.0521328449249268, -1.476157784461975, 0.8549903631210327, 0.00026436851476319134, 1.6886483430862427, -0.631766676902771, -0.459933876991272, 0.11885562539100647, 1.421950340270996, -1.6601016521453857, 0.6868173480033875, 0.9354282021522522, 0.5891185998916626, -0.8791043162345886, -1.992645025253296, 0.6213660836219788, -0.5021076202392578, 0.346451997756958, -0.33924663066864014, 0.6550530791282654, -2.325556755065918, 1.042755126953125, -1.002598762512207, 0.1030944287776947, -0.6897323131561279, 1.0877352952957153, 0.28663352131843567, -0.2540194094181061, 1.3205403089523315, 0.5767359733581543, 1.7704200744628906, -0.0006137426244094968, -0.32159724831581116, 0.9889538288116455, 1.014601230621338, -0.045432355254888535, 0.20707711577415466, 1.0505352020263672, -1.41946280002594, 0.2523869276046753, 0.5070240497589111, 1.689162254333496, 0.04012780636548996, -1.5356628894805908, -0.5016815662384033, -0.15103350579738617, 1.5283091068267822, -0.7656762003898621, -0.7217187285423279, 0.35166943073272705, -0.527263343334198, -0.09200936555862427, -1.639407753944397, 0.6743512153625488, -0.6830215454101562, 1.4764922857284546, 1.8017218112945557, 0.7858394384384155, -0.732430100440979, -0.9698979258537292, -2.2513420581817627, 0.4443144202232361, 1.0053106546401978, 0.8174597024917603, -0.9865568280220032, 1.4576027393341064, 0.9932203888893127, 2.623274803161621, 0.35310864448547363, 0.005685604643076658, -0.8471117615699768, 0.31412050127983093, -0.244114488363266, 0.5411203503608704, 0.16207285225391388, 0.18363606929779053, 1.402614951133728, -1.0190036296844482, -0.8481946587562561, 0.16644813120365143, 0.3839600086212158, 0.599884033203125, 0.6848488450050354, 0.8593006134033203, -1.3611702919006348, -0.3322925269603729, -0.09258171170949936, -1.7386614084243774, -1.2851682901382446, 0.6764654517173767, -0.40912848711013794, -0.8725326061248779, -0.5192233920097351, 1.465027093887329, -0.3740095794200897, 0.7079379558563232, 0.9527211785316467, 1.116950511932373, 0.49138227105140686, 0.5322197079658508, -1.2144132852554321, -1.5141206979751587, 0.7107658386230469, 0.7018623948097229, -0.8608418703079224, -0.8684476017951965, -0.49906429648399353, -1.4927160739898682, -1.0127339363098145, -1.2090847492218018, -0.7733616232872009, -1.3622190952301025, 0.08730143308639526, 0.11213140189647675, 0.7364306449890137, -1.1637109518051147, 1.2651292085647583, 0.8572754263877869, -0.35248783230781555, 0.8295688033103943, 0.24958235025405884, -0.6002476811408997, -0.3909183442592621, 1.496659517288208, -0.7541759014129639, -1.4824508428573608, -0.11926116794347763, -1.2544124126434326, -0.315577894449234, -0.6402113437652588, -1.345229148864746, -0.29472970962524414, -0.20594744384288788, 0.5705803036689758, 1.1804982423782349, -1.3876546621322632, -1.79313325881958, -0.11050401628017426, -1.2200357913970947, 1.3649789094924927, -0.008552244864404202, 0.8321132063865662, 0.10486792027950287, -0.6751068830490112, 0.28175392746925354, 2.5485541820526123, -0.4607081115245819, 1.6013559103012085, 2.2701406478881836, -1.513732671737671, -1.374271035194397, -0.9302383065223694, 1.429042100906372, -0.5110756754875183, -0.4847915470600128, 0.964155375957489, -1.1124070882797241, 0.5130704045295715, -0.17339523136615753, -1.0392683744430542, -0.32125991582870483, 0.13661152124404907, -0.7192914485931396, 1.1712687015533447, -0.4434581398963928, 0.22004517912864685, -1.8373254537582397, 1.459479570388794, 1.0045074224472046, 0.43420568108558655, -0.26456180214881897, 1.611624002456665, -0.44209423661231995, -1.2578572034835815, 0.851793110370636, 0.22371603548526764, 0.8179419040679932, -0.8138673901557922, -0.1826355755329132, 0.2962251603603363, 0.8987534046173096, -0.24985705316066742, 0.39491137862205505, 0.4809289574623108, 1.2017042636871338, -0.5982025861740112, 0.6312500834465027, 0.7532194256782532, 0.7912505865097046, -0.13374297320842743, 0.6315653324127197, 0.7981305718421936, 0.047885481268167496, 0.26981088519096375, -1.118748664855957, -1.61898672580719, 1.5350230932235718, -0.7876824736595154, 1.069628357887268, -0.4980427622795105, -0.8127934336662292, 0.40579068660736084, -0.12248202413320541, 0.1200452670454979, -0.5888509750366211, -1.5287848711013794, -1.2518506050109863, 0.824074387550354, 0.23956231772899628, -2.0347049236297607, 1.6516488790512085, 1.204033374786377, 1.4310771226882935, 0.06506102532148361, 0.25643235445022583, -0.24828478693962097, 0.5034016966819763, -1.9001243114471436, 0.3372647762298584, 0.045506954193115234, 1.3727216720581055, 0.4697503447532654, 1.2560518980026245, 0.03166348114609718, 1.8619327545166016, -1.292925477027893, 0.7510448694229126, 0.8430989980697632, -2.19708251953125, -1.030300498008728, -0.7188569903373718, 0.33007243275642395, 0.31919750571250916, 1.8680232763290405, 0.15128464996814728, -0.555805504322052, -0.4311334788799286, -0.41998055577278137, -0.44501060247421265, -0.1114371195435524, -0.1398296356201172, 1.0104563236236572, 1.6540874242782593, -0.37109899520874023, -0.428621381521225, 1.174797773361206, 0.27581167221069336, 0.694411039352417, 0.39742642641067505, 0.5407516956329346, 0.5463879108428955, -0.38265854120254517, -1.9166107177734375, -0.12402556091547012, -1.8111441135406494, -2.18514084815979, 0.9146283864974976, -0.2964908182621002, 0.1118805930018425, -1.4126214981079102, 0.5635522603988647, -0.4601317346096039, -0.31639474630355835, -0.5299219489097595, 0.06846866011619568, -1.3389854431152344, 0.5956881046295166, 1.255885124206543, -0.062088340520858765, -1.7823512554168701, 1.290271282196045, 0.667794942855835, -0.6531454920768738, 0.008821298368275166, -1.231087565422058, -0.37349700927734375, -1.8182079792022705, 0.05101538076996803, 0.07917458564043045, 0.5346445441246033, 1.0772403478622437, 0.6042462587356567, 0.8598992824554443, 1.6829147338867188, -1.1349616050720215, 0.44381585717201233, -0.38258984684944153, 0.05573781579732895, 2.0583724975585938, 0.6886851787567139, -0.3648255169391632, 0.0019582754466682673, 0.9680863618850708, 0.3274971544742584, -1.1894047260284424, -0.5293824672698975, -1.0601716041564941, 0.10719844698905945, 1.1331499814987183, 2.0417368412017822, 0.34090423583984375, 1.6332075595855713, 1.9923722743988037, -0.16217143833637238, -0.14249178767204285, 0.491604745388031, -0.7766324281692505, -0.4726234972476959, -0.0792132318019867, -0.36012741923332214, -2.111351728439331, 0.7934200763702393, -1.5961244106292725, 0.9781811833381653, 1.4065864086151123, -0.08324917405843735, -0.8697192072868347, 0.2684571444988251, 0.5496724247932434, 0.04320978745818138, -1.0766267776489258, 1.1705243587493896, -0.263260155916214, 0.05234145745635033, -0.20791272819042206, 0.5001490712165833, -0.19466334581375122, -1.1079667806625366, 0.9143161177635193, 0.5756622552871704, 0.5252264142036438, -0.7677343487739563, 0.3711593449115753, -0.9609014391899109, 0.315168172121048, 2.568197250366211, -1.690651297569275, 1.8047715425491333, -1.548427700996399, 0.8928147554397583, 0.35977309942245483, -1.0575387477874756, 1.8078926801681519, 0.17403271794319153, 0.42178478837013245, -0.8610917329788208, 1.1110564470291138, 0.5332505702972412, 0.2030046582221985, 0.2866564989089966, 1.177204966545105, -0.0535520538687706, 0.5108705163002014, 0.25061503052711487, 1.5897687673568726, -1.3107212781906128, -0.38795706629753113, 0.8124224543571472, 0.31639307737350464, 1.170849084854126, 0.8441045880317688, 1.097194790840149, -0.9338049292564392, 0.9448596239089966, -1.4347504377365112, -0.1213923990726471, -0.6163622736930847, -0.33517181873321533, -1.486521601676941, -0.9523868560791016, 0.2926260828971863, 0.09761366248130798, 0.5358151197433472, -1.088797926902771, -0.005446151830255985, 0.18347783386707306, 0.5500797033309937, -0.7426460385322571, 0.3768905699253082, 0.5863528251647949, 0.2452506721019745, 0.7539686560630798, 0.0826743021607399, -0.6852685213088989, -0.6112870573997498, -0.2857329249382019, 0.8601992726325989, 0.7944236397743225, 0.4214406907558441, 1.0633732080459595, 0.22467318177223206, -0.7582411766052246, 0.13057948648929596, -0.07714226096868515, 2.2359206676483154, 1.1962523460388184, 0.3824503719806671, -0.001395095488987863, 0.13950005173683167, -0.18734073638916016, -2.6855695247650146, -0.75162672996521, 0.3149203360080719, -0.26501864194869995, 0.009547654539346695, 0.48395106196403503, -0.27901819348335266, 0.10727802664041519, -2.162111282348633, 0.7034615874290466, 0.8439012169837952, 0.6780481934547424, -0.10673928260803223, 2.710988998413086, 0.09394620358943939, -0.9326626658439636, -1.2915312051773071, 1.1876099109649658, -1.379448413848877, 0.33959463238716125, -1.1405584812164307, 0.5271758437156677, 0.5381549000740051, -1.7174062728881836, -1.7408407926559448, 1.5249429941177368, 1.2072246074676514, 0.6024662852287292, -0.19708825647830963, -0.35778626799583435, 0.8148057460784912, 0.018087266013026237, 1.1517362594604492, -0.9348156452178955, 0.40435972809791565, 1.0054206848144531, -1.1231675148010254, -0.02064410038292408, 0.6666226983070374, -1.1682140827178955, 0.8751760125160217, -0.28274330496788025, -0.5502453446388245, -0.24194788932800293, 1.2729105949401855, -0.05436396598815918, 0.7154266834259033, 0.2636677324771881, 0.4507713317871094, 0.02356543019413948, -0.7171584963798523, -0.9209851622581482, 1.3721716403961182, 0.10999184846878052, -0.6965010762214661, -0.8721029162406921, -2.0203959941864014, 1.6376335620880127, 0.5085066556930542, -1.3770662546157837, -0.33766409754753113, 0.6661837100982666, -1.8576140403747559, -2.3963260650634766, -0.7237107753753662, -1.119547724723816, 0.38323304057121277, 0.46539899706840515, -0.5593995451927185, -1.0154926776885986, 0.10802362859249115, 1.4005787372589111, -0.8551382422447205, -0.9260643124580383, -0.038179442286491394, -0.631596565246582, 1.9521230459213257, 1.043501853942871, 0.018277158960700035, 0.7767065167427063, 0.820558488368988, 0.994515597820282, 0.30091848969459534, -0.146957665681839, 1.2350525856018066, 0.7572159171104431, 0.7030553221702576, -0.40247994661331177, 0.8536733388900757, 1.4363471269607544, -0.5795327425003052, -0.10180183500051498, 1.1116291284561157, 0.07954472303390503, -1.5767154693603516, 0.31231871247291565, 0.239903062582016, 1.0650651454925537, -0.8573436141014099, 1.2087129354476929, -2.6394786834716797, -1.0654270648956299, 1.5565359592437744, 0.5206748843193054, -0.3891974687576294, -1.148984432220459, -1.2556889057159424, 0.27002978324890137, 0.16589312255382538, -0.9966143369674683, 1.0749201774597168, -1.0195354223251343, -0.8658758997917175, 1.185896873474121, -0.49302756786346436, 0.034193217754364014, -1.5355361700057983, -0.4336059093475342, 0.17991888523101807, -0.2361225187778473, 1.9920817613601685, 0.866395115852356, -1.316815733909607, 0.5831220746040344, 1.7242997884750366, -1.0917506217956543, -0.2802061140537262, 0.9318723082542419, -0.3903568387031555, -1.437320351600647, 1.1419028043746948, -0.8563104867935181, 0.36132413148880005, 0.6352571845054626, -1.5580185651779175, -1.40501070022583, -0.6459410190582275, 0.29175063967704773, -0.7788046002388, -0.09100648015737534, 1.9660674333572388, -0.28131619095802307, -0.16291633248329163, 0.1501392126083374, -1.143034815788269, 0.22025001049041748, -0.6970939040184021, -0.5202307105064392, 0.7837325930595398, 0.21922868490219116, 1.6791040897369385, 1.5090751647949219, 0.23223045468330383, -0.2889246940612793, -1.162207841873169, -1.107962727546692, -0.5177205204963684, 0.8492917418479919, 0.3883286118507385, 0.7319071292877197, 1.9469927549362183, -0.6144473552703857, -0.17407609522342682, 0.13248072564601898, 0.0054255323484539986, -1.4916385412216187, -1.2156134843826294, -0.7786138653755188, -0.7652493119239807, -1.72587251663208, -1.0694974660873413, -0.5109362602233887, -1.4348241090774536, -1.383276343345642, 0.2211466282606125, -0.21922846138477325, -0.8257238864898682, 0.9391326904296875, 0.4781530201435089, -1.3901020288467407, 0.47043028473854065, 0.8869004249572754, 0.08274064213037491, -1.216226577758789, 2.03328537940979, -1.4825565814971924, -2.2870278358459473, -1.620837926864624, -1.874910593032837, 0.8084036111831665, -0.34566131234169006, -1.2222872972488403, -1.6989355087280273, -0.9855747818946838, 1.9706721305847168, 0.5998967289924622, 0.9709278345108032, -1.151250958442688, -0.4102938771247864, -0.3590032458305359, 0.7724061012268066, 0.9033600091934204, 0.18347834050655365, 1.2175545692443848, -0.5255011916160583, 0.3000618815422058, -0.8072311282157898, -1.5688300132751465, 1.0107080936431885, 0.18330053985118866, 0.9590647220611572, -0.8779551386833191, 0.36879172921180725, -1.6703952550888062, -1.3561841249465942, 1.2825987339019775, -1.293036937713623, 1.3781185150146484, -2.084315299987793, 0.04515910521149635, 0.7594895362854004, -0.4627481698989868, -1.5179945230484009, -0.6084747314453125, 1.0407817363739014, -1.6132264137268066, -0.782585620880127, -2.426335334777832, -1.915654182434082, -0.10413174331188202, 0.2416626214981079, 0.008552695624530315, -0.65449059009552, -1.3428386449813843, 1.1322317123413086, 1.415684461593628, 0.6434656977653503, -0.33916980028152466, 1.295438289642334, 0.5554851293563843, -1.457359790802002, 2.61676287651062, -0.569152295589447, 0.1370324045419693, -1.3187077045440674, 1.6386853456497192, -1.5517430305480957, 0.8480600118637085, -1.2516902685165405, 0.4883420169353485, -0.39652276039123535, 0.20050151646137238, 0.020876193419098854, -0.576087236404419, 0.49179404973983765, -0.7732087969779968, -0.8266811966896057, -0.5664485096931458, -0.24998921155929565, -1.8099008798599243, 2.81058406829834, -2.069383144378662, 1.0071402788162231, 0.6118438243865967, 0.5921840667724609, -0.7356996536254883, -1.66819167137146, -0.4497324228286743, -1.0116164684295654, 0.690918505191803, 0.7868353128433228, 0.697982132434845, -0.1916016936302185, 1.7134311199188232, -1.5092006921768188, -0.06934284418821335, 0.28546515107154846, -0.4964033365249634, 1.5568623542785645, 0.1457068771123886, 0.5211018919944763, -0.7652854323387146, 1.6447404623031616, 0.47559162974357605, -0.31644773483276367, -1.103795051574707, 0.883553683757782, -0.515746533870697, -0.7018536329269409, 0.09513388574123383, -0.12982065975666046, 0.5977595448493958, 0.035720013082027435, -1.122979760169983, 0.3732370436191559, 0.6293980479240417, 0.3653847575187683, -1.0504415035247803, -0.5693521499633789, 2.52807879447937, 0.30176669359207153, -2.069129228591919, 0.8724049925804138, 0.10023440420627594, 0.11354400217533112, 0.9215909838676453, -0.7673446536064148, 1.012518286705017, 2.3604698181152344, 1.126225471496582, 0.7857717871665955, -1.2396173477172852, 1.2869911193847656, -0.7901548743247986, 2.7327213287353516, -0.1702437549829483, -0.7413164377212524, -0.2660713493824005, -1.3953499794006348, 0.9072004556655884, -0.2720322012901306, 0.8696338534355164, -0.5263110995292664, -0.5308700799942017, -0.8062466382980347, -0.7696232795715332, 1.491154432296753, 0.3683389127254486, 0.5469653606414795, -1.1517122983932495, 1.1229877471923828, 1.7018699645996094, 0.871228039264679, -0.6381749510765076, 1.4577280282974243, 0.4491705298423767, 1.6515589952468872, 0.11519868671894073, 0.614093542098999, 0.7087120413780212, 0.61836177110672, 0.18357491493225098, -0.4908657371997833, 0.8493796586990356, -1.2773220539093018, -0.27854838967323303, -0.5463396310806274, -0.4188667833805084, -0.48129725456237793, 0.30578044056892395, 1.3718950748443604, 0.28686219453811646, 0.4517750144004822, -0.6748446822166443, -0.017627857625484467, 0.2282032072544098, 0.5508683919906616, -2.036482810974121, 0.8997964859008789, 1.828679084777832, -0.3722038269042969, 1.3678998947143555, -0.3981914222240448, 0.1964448094367981, 0.30459558963775635, -0.969634473323822, -0.5290675759315491, -0.6015666723251343, 0.8866575956344604], [0.35135647654533386, -1.563888669013977, 0.9067424535751343, 0.4453625977039337, 0.2668255865573883, 0.4147166311740875, 1.5941706895828247, -0.5888621807098389, 1.8233951330184937, -0.7357555627822876, -1.39927339553833, 0.9232004284858704, -0.733998715877533, 0.4842056334018707, -1.264600396156311, 0.6273999810218811, 0.8182165622711182, 1.4721697568893433, -0.27109450101852417, 0.029952233657240868, 0.8873268365859985, 1.1394875049591064, -1.3340702056884766, 0.37920480966567993, 0.2116699516773224, 0.2175951451063156, 0.6459560990333557, -0.4147338271141052, -1.557270884513855, 0.9563705325126648, -0.5496764779090881, -0.46515733003616333, 2.0924153327941895, 0.7206990718841553, -0.49691084027290344, -0.2210422307252884, -1.4312083721160889, 0.09344151616096497, 0.5543599724769592, -0.510560154914856, -1.1918034553527832, -0.631316065788269, -0.9607384204864502, -1.1749780178070068, 0.6138749122619629, 0.12199493497610092, -0.7950233817100525, 2.369263172149658, 0.478292316198349, 0.906351625919342, 1.3873224258422852, -0.10689358413219452, 0.11268774420022964, 0.9186592698097229, 0.2714863121509552, -0.012102876789867878, 0.3052971661090851, 0.7559676170349121, 1.5504480600357056, -0.5136964321136475, 0.10435766726732254, 1.9061346054077148, -0.6301428079605103, 1.0777428150177002, -3.304335117340088, -0.8550859093666077, -0.6705144047737122, 1.574729323387146, -0.0825335681438446, 0.8844273686408997, 0.5222314596176147, -0.34186631441116333, 1.3031830787658691, -1.1149433851242065, -0.45069220662117004, -0.09231250733137131, -0.4097045660018921, 0.5456780791282654, -0.6758650541305542, -1.0490963459014893, 1.35183846950531, 0.5402542352676392, -0.2551974952220917, -0.09461954236030579, -0.9236311912536621, 0.05032522976398468, 1.540661096572876, 1.2748063802719116, 0.7389183640480042, -0.9064459800720215, 1.4340192079544067, 0.35786062479019165, 1.183448076248169, 0.8911120295524597, 0.3979020416736603, -0.07710756361484528, -0.30446431040763855, -1.7754526138305664, -1.5767459869384766, -1.6682733297348022, -0.04182509332895279, 0.07186438143253326, 0.9129976630210876, -0.4505317211151123, 1.6808316707611084, -0.8040751218795776, 0.4786432385444641, 1.358333945274353, 0.897265613079071, 0.0280335433781147, 1.4586787223815918, 0.7297483086585999, -1.097008466720581, 0.9325587153434753, 1.1360547542572021, 0.8844255805015564, 0.4846290349960327, 0.4267336428165436, 0.8419685363769531, -0.35153666138648987, 1.3972153663635254, 1.3458179235458374, 1.3216238021850586, 2.7545671463012695, 0.4663143754005432, -0.8341291546821594, 0.5821681022644043, 0.2092951238155365, 0.4477172791957855, -0.7738370299339294, -0.46102505922317505, -1.1492019891738892, 1.720200538635254, 0.2986326217651367, -0.9987150430679321, -0.2523155212402344, 0.3912622332572937, 1.1494991779327393, 2.02335524559021, -1.1315548419952393, -0.8060942888259888, -0.09939392656087875, 2.637416362762451, -0.8209974765777588, -0.822288453578949, -0.9084614515304565, -1.932586431503296, 0.05687341094017029, -0.13346439599990845, 2.1636247634887695, 0.2914750874042511, -1.6895573139190674, -1.3837106227874756, 0.98716139793396, -0.058247003704309464, 0.02583426982164383, -1.2581228017807007, 0.10279256105422974, 1.799538493156433, -0.4561232924461365, -1.2050443887710571, -1.7928065061569214, -0.17141442000865936, -0.8899076581001282, 0.9155732989311218, 0.42736226320266724, -0.3835069537162781, 1.2061190605163574, -0.1310446709394455, 1.084384799003601, -0.19330763816833496, 1.040966510772705, -1.1716184616088867, -0.11255433410406113, 0.06600597500801086, -0.2584066092967987, 0.7969887852668762, 0.10106945782899857, 0.3013412654399872, 0.15490321815013885, -1.181014895439148, 0.9944806694984436, -0.6238706707954407, 0.36405977606773376, 0.054557379335165024, -0.027904871851205826, -0.11239651590585709, -0.02306930348277092, -1.3090013265609741, 0.0945492684841156, -0.17850247025489807, -1.1255136728286743, -1.122387170791626, 0.9969292283058167, 1.1936146020889282, 0.2056238204240799, -1.3169370889663696, -0.3667948544025421, -0.18624359369277954, 0.5246334671974182, 1.4579660892486572, -0.5944966077804565, 1.8183542490005493, 0.7731481194496155, 0.7343911528587341, 1.1023825407028198, 0.2987978458404541, -1.5349441766738892, -1.9551787376403809, 0.2995472550392151, 0.46127262711524963, 0.407821387052536, 0.0038855315651744604, -1.1752904653549194, 0.13886824250221252, 1.3306246995925903, -0.11874814331531525, 1.6003628969192505, 0.12779679894447327, -0.6798902750015259, 0.5750163793563843, 1.0097101926803589, -0.8618857860565186, 3.313720464706421, -0.11767806857824326, -0.8589448928833008, 0.5588319301605225, 0.5910352468490601, 2.4243130683898926, -1.2923798561096191, 0.6741884350776672, -0.7842074632644653, -0.8826335668563843, -1.0033760070800781, 0.13879592716693878, 0.9575163125991821, 0.3263731598854065, 1.5402719974517822, 0.06987373530864716, -0.17111551761627197, -0.8795783519744873, -0.3423067033290863, 1.5695676803588867, 0.5033875703811646, -0.10466086119413376, 1.1827189922332764, -0.35861167311668396, 0.3605099022388458, -0.31941086053848267, -2.141029119491577, -1.5599157810211182, -1.4090209007263184, 0.8986973166465759, 0.47158199548721313, 0.5481517910957336, -1.9503381252288818, 0.8177282214164734, 0.6431021094322205, 1.1641793251037598, -1.001368761062622, -0.22844406962394714, 0.5051858425140381, -0.6522120237350464, 0.5518333911895752, -0.7739623188972473, -0.762396514415741, 0.23721453547477722, 2.141939401626587, 0.5059114098548889, -0.2603999674320221, 0.02782393991947174, 1.5107563734054565, -0.08887021988630295, 0.3803540766239166, 0.9634825587272644, -1.2296923398971558, -0.971171498298645, 0.6354478597640991, 0.9382128715515137, 1.0090525150299072, 0.14233499765396118, 0.5783911943435669, 0.1597847193479538, -1.5296772718429565, -0.5405264496803284, -0.9029656052589417, -0.21499784290790558, 0.12021327763795853, -0.5924678444862366, 0.1805034577846527, -0.5147777795791626, -1.642890214920044, 0.5922450423240662, -2.198885679244995, 1.0403497219085693, 1.122197151184082, 0.3786376416683197, 1.3812589645385742, 0.20925170183181763, -0.022584358230233192, 0.8519396781921387, 0.33684927225112915, -1.0706334114074707, -0.12329177558422089, 1.162776231765747, 0.9388555288314819, -1.4259790182113647, -0.8904405832290649, 1.732393503189087, -0.7324092388153076, 0.05246752128005028, -0.08178212493658066, 0.3554585874080658, -0.31364157795906067, 0.900493323802948, -1.76254403591156, -2.188977003097534, 1.1216650009155273, 1.013554334640503, -0.012213696725666523, -0.3036355674266815, -1.0447214841842651, 0.39335915446281433, 1.0855944156646729, -1.8289638757705688, -0.17210820317268372, 0.2951616048812866, 1.1663596630096436, 0.3383161127567291, -0.4333007335662842, 0.627934992313385, -1.4265915155410767, -0.7963212132453918, 0.6881741881370544, 1.5585284233093262, -0.14563241600990295, -1.2120999097824097, 0.14409296214580536, -0.4631769061088562, -0.09725045412778854, 0.19584549963474274, 1.3590065240859985, -0.1741202026605606, 1.102547526359558, -0.2582199275493622, 0.9474391341209412, 0.26703178882598877, 0.5369959473609924, 1.5335205793380737, -1.2346704006195068, 0.49075257778167725, 0.05727612227201462, -0.1450486034154892, 0.6286242008209229, -0.06001405790448189, -1.277527928352356, 0.08824177086353302, 0.7079474329948425, -0.6668813228607178, 0.7682176828384399, -0.7473629117012024, -0.30330562591552734, 0.04313238710165024, 1.2292221784591675, -0.10932974517345428, -1.2456051111221313, -1.7160372734069824, 0.5977122187614441, -2.107102394104004, 0.6871219277381897, -0.4503193795681, -0.6582292318344116, 2.3010268211364746, -1.0158003568649292, 0.3885483145713806, -0.35043299198150635, 1.3062525987625122, -1.5273420810699463, 0.937707245349884, 0.7529334425926208, -0.4579700231552124, -0.5783859491348267, 1.7983667850494385, 1.0088508129119873, 0.9421405792236328, -0.30473920702934265, 1.0404303073883057, -1.3116360902786255, -1.584810495376587, 0.38177043199539185, 0.20687954127788544, -0.22841855883598328, 0.6900837421417236, 0.7916862368583679, 1.0583021640777588, 0.2295025885105133, 1.3975874185562134, 0.4867766499519348, -0.7589333653450012, 0.7282017469406128, 0.7861219048500061, 1.3448436260223389, 0.5261484384536743, 0.043707624077796936, -0.737338662147522, -0.22821581363677979, -2.695509672164917, 0.5373499393463135, 0.26285019516944885, 1.9011462926864624, -0.3711225688457489, 3.232255220413208, 1.368517518043518, -0.6639588475227356, -0.7779281139373779, 1.7281672954559326, 0.07149197161197662, -1.4658682346343994, -0.7303832769393921, -0.20571987330913544, 0.4710526466369629, -0.13586309552192688, -0.6351395845413208, -0.8408753275871277, 2.401193618774414, 1.32044517993927, 1.7279340028762817, 1.6951067447662354, -3.1874747276306152, -0.32680463790893555, 0.6411588191986084, -1.4116852283477783, -0.6537282466888428, -2.363365411758423, -0.9274132251739502, 0.7685948014259338, -1.21396803855896, -1.0200157165527344, 0.7920384407043457, -0.32061174511909485, 0.37626901268959045, -0.41284435987472534, -0.1798659861087799, -1.346314549446106, 0.2723490595817566, -2.1188008785247803, 0.1642334759235382, 2.174874782562256, 1.297149896621704, 0.6007359027862549, 0.4964020252227783, 0.7562782168388367, -0.5579074621200562, -0.7893428802490234, -1.8824462890625, 0.6428395509719849, -0.7090108394622803, 0.8098295331001282, -1.0247604846954346, -0.1780160814523697, -0.13000889122486115, -0.4780312478542328, -1.1477159261703491, -1.3398147821426392, 0.19603779911994934, 0.25200462341308594, -0.6799142360687256, 0.813256025314331, 1.646779179573059, 0.39208248257637024, -1.4883157014846802, -0.5867620706558228, -0.5226767659187317, -0.7121421694755554, -0.17095911502838135, 0.09887213259935379, 0.03671807423233986, 1.4932054281234741, -3.3294312953948975, -0.43436571955680847, 0.0811488926410675, 0.8202522993087769, 0.4314766526222229, 0.21737977862358093, -1.0033588409423828, -1.560843586921692, -1.044360637664795, -1.3739204406738281, -0.5740563273429871, 1.928320288658142, 0.48304036259651184, 0.8462570905685425, 0.30554133653640747, -0.020001742988824844, -0.0710049718618393, 0.5288541316986084, 0.6021560430526733, 1.3012176752090454, 0.39457884430885315, 0.6275277733802795, -0.4427890479564667, 0.21831770241260529, -0.243605375289917, -1.1043673753738403, -0.2914344370365143, -0.47179561853408813, -1.66885244846344, -0.5622192025184631, -0.8206093311309814, -0.2852412164211273, -0.1331373006105423, 1.006853461265564, -1.2741678953170776, 0.4346565008163452, -0.7840988636016846, 0.09810113161802292, 0.048048049211502075, -0.32105961441993713, 0.44655248522758484, -1.8643443584442139, 0.8036258816719055, 0.23573079705238342, -1.0721383094787598, 0.40673914551734924, -0.15237505733966827, 0.5120308995246887, 0.45342838764190674, -0.17236685752868652, 0.5155487060546875, -0.48221227526664734, -1.0305448770523071, 0.5102971196174622, 0.13062657415866852, 0.14246509969234467, -0.27109915018081665, 1.1747792959213257, -0.537574827671051, -0.7990022897720337, -1.3096057176589966, -0.6941412687301636, -1.428687572479248, -0.2028873711824417, -0.7932199835777283, 0.6822628378868103, 0.8325898051261902, -1.7831889390945435, 1.3221083879470825, 1.39222252368927, 0.637087881565094, -0.9611112475395203, -0.040842246264219284, -0.16527020931243896, 1.1420053243637085, -0.12733764946460724, 1.8640644550323486, 0.583006739616394, 1.106689214706421, 0.17223738133907318, -2.208744764328003, -1.378922700881958, -1.5523089170455933, 0.1585407257080078, -0.5700945258140564, -1.4622431993484497, 1.0449292659759521, -0.6146328449249268, 0.17888197302818298, 1.6231045722961426, 2.0935401916503906, -0.35106223821640015, -0.28773099184036255, 0.03438837081193924, -0.9983033537864685, 1.4651817083358765, 0.002538906643167138, 1.6422064304351807, 1.1578409671783447, -0.324166864156723, -1.5133477449417114, -1.734676718711853, -1.0506888628005981, -0.4234331548213959, 0.9726560115814209, 2.1013243198394775, 0.7996013760566711, 0.0258804801851511, 0.3680521249771118, 0.5840084552764893, 0.36638474464416504, -0.49268007278442383, -0.6608516573905945, 1.51703941822052, 1.0060889720916748, 0.7505266666412354, -0.29334554076194763, -1.4109104871749878, -0.4352550208568573, 0.5145377516746521, 1.9993748664855957, 0.7512397170066833, -0.25748106837272644, -0.5901182293891907, 1.2766613960266113, -1.6983904838562012, -0.30855345726013184, 0.03824221342802048, 1.0758321285247803, -0.5541655421257019, 1.501098394393921, -0.9000678658485413, 0.5055769085884094, -0.42262881994247437, -0.025700923055410385, 0.22022061049938202, -0.8362274169921875, -0.983890950679779, -0.39357584714889526, -1.5219120979309082, -0.7913004159927368, 0.4713144600391388, -2.4545140266418457, 1.0311810970306396, -0.51478511095047, 0.31382259726524353, 1.2372397184371948, 2.360147476196289, 2.0833563804626465, -0.10690544545650482, 0.33256208896636963, -0.20705653727054596, 0.22844645380973816, -0.03996659070253372, 1.3807204961776733, 0.1702115535736084, 0.4415716230869293, -0.09880369901657104, 1.490383505821228, 0.5928872227668762, -0.12246391177177429, -0.6061626672744751, 1.2898781299591064, -0.9298503398895264, -1.7322129011154175, -0.021943479776382446, -0.3914611041545868, -0.456216037273407, 0.945253312587738, -0.34182554483413696, 0.6988082528114319, 0.06424546986818314, 1.2881999015808105, -0.9495763778686523, 0.2028452754020691, -0.3357090651988983, 0.4673965275287628, 0.5479069948196411, -0.8976897597312927, -1.3186787366867065, -0.9548512697219849, 1.092485785484314, -0.7760087251663208, 0.9515079259872437, -1.6526211500167847, 0.5476506948471069, -0.9918131232261658, -1.027221918106079, -0.7910131216049194, 0.05250582844018936, -0.7150431275367737, -0.5348557233810425, 0.6615299582481384, -0.8594937324523926, -0.5940277576446533, -0.31297820806503296, -0.020377211272716522, -0.9831671118736267, 0.6269820928573608, 0.169722318649292, -1.4634209871292114, 0.01475328579545021, 0.9897229671478271, 0.8752773404121399, -1.3659112453460693, -0.7538003325462341, 1.027193307876587, -1.5578422546386719, -0.8186719417572021, 0.15039633214473724, -2.50834321975708, -0.3237350583076477, -1.9943128824234009, -0.927217423915863, -0.426487535238266, 0.6477085947990417, 1.6846858263015747, -0.43195587396621704, -1.7253966331481934, 0.734184205532074, -0.2999342083930969, 0.6518740057945251, -0.7570278644561768, 0.48118796944618225, -0.7307557463645935, -2.0876224040985107, -1.5341687202453613, -1.5462620258331299, 0.5568972229957581, 1.1778192520141602, 0.6739094853401184, 0.2969413995742798, 1.7399085760116577, -0.38136282563209534, 0.2376454770565033, 1.1657960414886475, 0.15467539429664612, 0.06437113881111145, -0.42742255330085754, 0.375126451253891, -0.8188722133636475, 1.064825415611267, -0.0716095045208931, 0.5971664786338806, -1.7952111959457397, -0.42755305767059326, 0.3322305381298065, -0.8184129595756531, -1.0072710514068604, 0.5282845497131348, 0.6701710224151611, -0.9903203248977661, -2.0291941165924072, 0.428923636674881, 1.8272571563720703, -1.3320896625518799, -0.08827225863933563, -1.3645374774932861, -1.550957441329956, -0.7124029397964478, -0.42751312255859375, -0.32878923416137695, 0.30298590660095215, 1.7210222482681274, -0.43151015043258667, 0.47636690735816956, -0.9655483961105347, 0.5570688843727112, -0.5248439908027649, -0.5331992506980896, -0.21589398384094238, 0.0544244684278965, 0.24961233139038086, -0.3171273469924927, 0.7873135805130005, -1.0091644525527954, -0.003272568341344595, -0.20531246066093445, -0.9408585429191589, -1.1235508918762207, -0.25516578555107117, -0.09656592458486557, 1.2770074605941772, 0.8621150255203247, -0.7097768187522888, -1.6948415040969849, -0.12560245394706726, -1.603278636932373, 0.298794150352478, -1.3891148567199707, -0.09688752889633179, 1.462099313735962, -0.1406150907278061, 1.0475261211395264, 0.06223030015826225, -0.32690155506134033, -1.198652744293213, -0.5749805569648743, -0.17472213506698608, -1.1893969774246216, -0.2690902054309845, -1.4176557064056396, -0.8154472708702087, 0.1181938424706459, 0.40623417496681213, 1.7011971473693848, -0.09395419806241989, -0.4280177652835846, -0.4890082776546478, 0.03265470638871193, 0.06980477273464203, 0.8791359663009644, -0.39838463068008423, -1.2870609760284424, 0.2188541144132614, 1.1598732471466064, -0.8809648156166077, -1.4327582120895386, -1.527195692062378, 0.6258779764175415, -1.6735855340957642, -0.1890971064567566, 0.061487384140491486, -2.0257441997528076, -0.10748607665300369, 0.21596455574035645, 0.30530431866645813, -1.4247082471847534, -0.055775754153728485, -1.649652361869812, -0.1631762534379959, 0.249597430229187, 0.3314211666584015, -0.41120409965515137, -0.5746873021125793, 0.1509939432144165, -0.42434918880462646, 1.7621103525161743, 1.6507350206375122, 0.746816098690033, -0.16516029834747314, 0.38376060128211975, 0.10285889357328415, 1.1314202547073364, 1.0822436809539795, 0.2513154447078705, -0.07364717870950699, -0.7103727459907532, -0.9706464409828186, -0.6327600479125977, -0.8768954873085022, 0.40281981229782104, -0.7422428131103516, -0.5390576124191284, 0.2133546769618988, 0.15885238349437714, 0.8879261016845703, 0.9786081314086914, -0.4521431624889374, 0.0448041595518589, -2.228351593017578, 0.5094712376594543, 0.05193896219134331, -0.2962183654308319, -2.1429190635681152, 0.7239806056022644, -0.49287334084510803, 0.7975435256958008, -0.8171983957290649, -2.90419864654541, -0.5843676924705505, 0.4920056164264679, -1.2951068878173828, -0.7150062918663025, 0.7405582070350647, -0.7329207062721252, 0.08966002613306046, 1.2113739252090454, 0.2408897876739502, 0.24144893884658813, -2.7393240928649902, 0.631294846534729, 0.3475903272628784, -1.5049837827682495, -0.1723197102546692, -1.3764915466308594, -0.17867562174797058, 0.23576706647872925, 0.11346295475959778, 1.1773265600204468, -0.9547451734542847, 0.35049471259117126, 0.8748089075088501, 0.07005376368761063, -0.6050265431404114, 0.4313000738620758, -1.1776525974273682, 0.2461864948272705, -0.7441040277481079, 1.6110872030258179, 0.19791847467422485, -0.7115534543991089, -0.26677533984184265, -0.020801281556487083, 0.8233745098114014, -0.12843596935272217, 1.8256508111953735, -0.50075763463974, 0.853695273399353, -0.3710658848285675, -0.6742216348648071, -1.608106017112732, 0.33428868651390076, 0.10697994381189346, -0.03713251277804375, -0.2702219784259796, 0.8908015489578247, 0.5605567693710327, 0.4529215693473816, 0.24253372848033905, 1.7836897373199463, -2.3023645877838135, 0.6728826761245728, -0.7904744744300842, 1.159294843673706, 0.024067696183919907, -1.0123021602630615, -0.24267537891864777, -1.3917267322540283, 0.5535426139831543, 1.025811791419983, 0.49077165126800537, 0.5149977803230286, -1.5824271440505981, 0.573775589466095, -0.7489458918571472, -0.600908637046814, 1.4166055917739868, 1.8523247241973877, -0.4593846797943115, 1.2169995307922363, -0.9913035035133362, 0.8528419733047485, -0.46628129482269287, 0.22298599779605865, -0.6998827457427979, 1.4439095258712769, 0.8599912524223328, 2.4592204093933105, 0.2970527708530426, 0.4415656626224518, 1.7710192203521729, 1.26964271068573, -0.6840724945068359, 1.1991742849349976, 0.25282707810401917, -1.2407557964324951, -0.39479726552963257, 0.30798569321632385, -0.9469929337501526, 1.141903042793274, 0.6169874668121338, 0.6421657204627991, 0.9090502262115479, 0.5353545546531677, 1.0172131061553955, -0.08960290253162384, 1.265002727508545, -0.37730666995048523, -0.8359696865081787, 1.0631879568099976, -1.2738581895828247, -0.5334904789924622, 0.04467002674937248, 0.3642294406890869, -0.5472478866577148, -1.2980607748031616, 0.5808653831481934, -1.7244303226470947, 0.7006822824478149, -0.6007091999053955, -1.2296007871627808, -0.24579066038131714, -0.7152159214019775, -0.10058245807886124, -1.6937148571014404, 1.2165571451187134, -0.3377074897289276, -0.6957181692123413, -0.7419867515563965, 0.48063305020332336, -0.3741769790649414, -1.087701439857483, 0.6486985683441162, 1.9468578100204468, -2.0957860946655273, -1.1790719032287598, -0.26961055397987366, 1.1881319284439087, -1.0459283590316772, 0.5461781620979309, 0.3118683397769928, -0.5070663094520569, -1.1914154291152954, -1.0018635988235474, -0.3653784692287445, -1.8012733459472656, -0.22097435593605042, -0.6558403968811035, -1.552321195602417, -0.1810792088508606, -0.9488363265991211, -0.0860084742307663, -0.3041471242904663, -0.8402459621429443, 0.9730401635169983, -0.6148754358291626, -0.341659277677536, 0.32503825426101685, 1.0924047231674194, -1.4090392589569092, 1.3833608627319336, -0.5269597172737122, -1.0386090278625488, -0.08631180226802826, -1.5922517776489258, -0.254204124212265, 0.43868857622146606, -0.23901090025901794, 1.0307083129882812, 1.4416431188583374, 0.6285378336906433, -1.0312834978103638, 0.20435844361782074, 0.6832913160324097, 2.4944958686828613, 0.5124019980430603, 0.7851547598838806, 0.8508804440498352, 0.628055214881897, 0.4200020432472229, 1.4186582565307617, -0.8086726069450378, 0.01890712045133114, 1.5229110717773438, -1.7634155750274658], [1.8363206386566162, 0.5152472853660583, -0.3722262978553772, -2.331718921661377, -0.999323308467865, 0.006978052668273449, 1.1415586471557617, 0.2894606292247772, -0.21478068828582764, 0.49705302715301514, -0.6916227340698242, -1.4527602195739746, -0.375007688999176, 1.6693001985549927, -0.386120080947876, 0.11235824972391129, 1.5914908647537231, -0.2187267392873764, 0.8678355813026428, 1.435897946357727, 2.299389362335205, 0.007323139347136021, 0.6313879489898682, 0.5255900621414185, -0.11377062648534775, 1.1541001796722412, 0.29439109563827515, 0.008854986168444157, 0.5492678284645081, -1.2513666152954102, 2.5590896606445312, 0.5933043360710144, 0.006667620502412319, -0.5438380837440491, -0.3226463496685028, 0.2574668526649475, -2.13334584236145, -0.3042892515659332, -0.0338333323597908, -1.1511709690093994, -3.097886323928833, -0.024130655452609062, -0.7493164539337158, 1.4376659393310547, -0.3418225646018982, 0.7551189064979553, 1.0233409404754639, -2.824803113937378, 0.20715536177158356, -1.1511377096176147, 1.1550395488739014, 0.05708092451095581, 2.1817543506622314, 1.2759875059127808, -1.5052978992462158, 0.9061184525489807, -1.146101951599121, 1.6244255304336548, -0.3408787250518799, 1.2593071460723877, -0.6463391780853271, -0.40561431646347046, 1.0509546995162964, -0.1587238609790802, -1.4777108430862427, 0.5240551829338074, -0.7803448438644409, 0.5134572386741638, 0.14881564676761627, -1.2544564008712769, -0.01600109413266182, 0.15457159280776978, 0.7479020357131958, -0.7908331155776978, -0.7227110862731934, 0.7159419059753418, -0.3507571816444397, -1.368952989578247, -0.5779494047164917, 0.5141011476516724, -1.0921798944473267, -0.8455915451049805, -0.8919959664344788, 1.2284796237945557, 0.9259179830551147, -0.6165624260902405, -1.6460474729537964, 0.026474706828594208, -1.9065220355987549, 1.107396125793457, -1.1665213108062744, 0.9297404885292053, -0.19081483781337738, 1.0507371425628662, -0.41801801323890686, 0.47775641083717346, -0.9170913696289062, -1.0846298933029175, -1.5714828968048096, -0.6925016641616821, -0.9460936188697815, -0.21266217529773712, 0.5273547768592834, 1.3425718545913696, -0.048974525183439255, 0.17449021339416504, -0.960677444934845, -0.5628608465194702, -0.3939206004142761, -0.5204963088035583, 1.7450228929519653, 1.5493659973144531, 0.3122163712978363, -1.1246533393859863, 0.3335533142089844, 1.8710860013961792, 0.7788712382316589, -1.0263477563858032, -0.8648272156715393, -0.7182096242904663, 0.7444106340408325, -0.3056439161300659, 0.7380036115646362, -0.26326245069503784, -2.118600368499756, -2.2100913524627686, -0.8870897889137268, 0.12918058037757874, 0.4993388056755066, 0.43232616782188416, -1.3493307828903198, 0.09441307187080383, 0.8769993185997009, -1.7627887725830078, 0.09818637371063232, -0.5692047476768494, -0.5815271735191345, -1.9001578092575073, -0.46869146823883057, 0.23410314321517944, -0.15149566531181335, 0.05326772481203079, -1.1690698862075806, 0.920852541923523, 0.49502837657928467, -1.642749547958374, 0.12701067328453064, -0.517147421836853, -0.3641875386238098, 0.971915066242218, -0.04028230533003807, -0.8867668509483337, -0.6992139220237732, 1.694468379020691, -1.7206003665924072, 0.08005466312170029, 0.37502285838127136, 0.113023541867733, 1.2800779342651367, 0.8309358954429626, -0.5338500142097473, -0.7593127489089966, -0.9677706956863403, 0.7342425584793091, -0.2672392427921295, 1.3786412477493286, -2.456596851348877, 0.2995828092098236, 0.827803373336792, -0.2872464954853058, 0.5567391514778137, -0.26211321353912354, 0.4916619062423706, 0.26335883140563965, 1.0828238725662231, 2.407850503921509, 0.6809871792793274, -0.929438591003418, 1.150862693786621, -0.6079918742179871, -0.08639881759881973, 0.26590046286582947, 0.17646627128124237, 1.1922636032104492, 0.30120471119880676, 0.8226351737976074, 0.44738394021987915, 1.4778295755386353, 0.13526299595832825, -0.6811478734016418, 1.623416543006897, 0.5621272325515747, 1.8813427686691284, -0.651487410068512, -0.9417412281036377, -2.595963478088379, 0.7044137716293335, -0.12238271534442902, -0.10290691256523132, -0.9787641763687134, -0.10696620494127274, 0.027020713314414024, -1.2677534818649292, -0.22993983328342438, -0.5784164667129517, -0.16503013670444489, 0.632569432258606, -0.6646847724914551, 1.3491517305374146, 0.8109406232833862, -0.6899400353431702, 0.07091411203145981, 0.22188396751880646, 1.2916994094848633, -0.036941833794116974, 0.8546714186668396, -0.48496493697166443, -0.38311147689819336, -1.2652846574783325, -0.5285668969154358, 0.09411126375198364, 0.8329129815101624, 1.6530333757400513, -1.5260190963745117, -0.38583123683929443, -1.5870522260665894, -0.9656140804290771, -0.15484881401062012, 0.7999873161315918, -0.4929561913013458, -1.121593713760376, -0.9565628170967102, -0.8709527254104614, 0.9280861616134644, 1.1921297311782837, -2.7371323108673096, -0.8377745747566223, 1.496856451034546, 2.079120397567749, -1.146010160446167, 0.8927673101425171, -0.5143033862113953, 1.3455536365509033, -1.0003479719161987, -0.5294645428657532, -0.8533638119697571, -0.02397402748465538, 2.1830034255981445, -0.5036578178405762, -1.699095368385315, 0.378698468208313, 0.9642844200134277, -0.25529661774635315, -0.7763352394104004, 0.14220426976680756, -1.120863676071167, 0.21168039739131927, 0.9024772047996521, -0.35381975769996643, -1.4442260265350342, -1.3902872800827026, 0.06439200043678284, -1.3469898700714111, -2.2784476280212402, 0.6015898585319519, -0.0696730986237526, 0.4832107424736023, 1.0404222011566162, -0.5647963285446167, -0.5235297083854675, 1.7957489490509033, 1.5722635984420776, 1.5034480094909668, -0.6535002589225769, 1.3661525249481201, 0.2100018560886383, -0.4687528908252716, -0.34028616547584534, 0.8442948460578918, -0.12978588044643402, 0.9441298246383667, -0.6707834601402283, 0.15024276077747345, 1.1912511587142944, 1.139899730682373, 1.5023980140686035, -1.3225382566452026, 0.18124480545520782, -2.0689613819122314, -0.3523220121860504, 1.1849626302719116, -1.6763006448745728, -0.3381395637989044, -0.9901496171951294, -0.514374315738678, 1.409453272819519, -0.4549523591995239, 0.14839881658554077, -0.7053993344306946, 0.01028821524232626, 0.568966269493103, -0.3047499656677246, -0.593632698059082, -0.6321133971214294, 0.2837812006473541, 0.16607047617435455, -0.34522706270217896, -2.528831720352173, -1.5112271308898926, -1.28187096118927, 0.14768460392951965, 0.8108049631118774, 1.002456784248352, 0.6117913722991943, -0.5108228921890259, -0.7800987362861633, 1.2786011695861816, 0.2562794089317322, 2.1299827098846436, -0.21583876013755798, 1.3412185907363892, 0.6676339507102966, -0.17594437301158905, 0.7283041477203369, -2.0807247161865234, 0.10610776394605637, 0.21043236553668976, 0.5711843371391296, 1.0781745910644531, 0.6719567775726318, -1.616686224937439, 1.4108227491378784, 1.229032039642334, -0.5598505139350891, -1.4762636423110962, -1.1624265909194946, 0.6822589039802551, 1.1837563514709473, 1.2235316038131714, 0.6450462937355042, -0.10057167708873749, -0.2724020481109619, -1.2451194524765015, 0.053497713059186935, 0.5727764964103699, 0.3053348660469055, 0.6650635600090027, 0.254852294921875, 1.1728112697601318, 1.7828150987625122, 0.8881763815879822, 0.5868117213249207, -1.0271868705749512, -1.109731912612915, -1.1258416175842285, 1.4246060848236084, -0.03654288128018379, -0.657516598701477, -0.10154590755701065, -1.0636887550354004, -0.16385559737682343, -0.2894226014614105, 1.1557964086532593, -0.6275647282600403, 0.32678163051605225, 0.8316317796707153, 0.14125922322273254, 0.12823386490345, 0.45373889803886414, 0.7226503491401672, -0.5606879591941833, -2.752133369445801, 1.457396149635315, 0.6309195160865784, 0.26771342754364014, 0.5307497382164001, -1.3314567804336548, -0.3030107915401459, -1.7851839065551758, 0.1738263964653015, -0.0679946169257164, 1.1251202821731567, 1.2932372093200684, -0.6581907868385315, -0.6022473573684692, 0.0024782009422779083, -1.980165958404541, 2.3661701679229736, -1.96200692653656, -0.45940911769866943, 0.16316446661949158, 0.7112177610397339, 1.3474808931350708, -0.9018153548240662, -0.6455134749412537, 0.33920371532440186, 0.4830686151981354, -0.847477912902832, 0.24690411984920502, 0.25402092933654785, -1.6179592609405518, 1.0063005685806274, -1.532991886138916, 0.24128635227680206, -0.0739823654294014, -0.8338231444358826, 1.1672202348709106, -1.3241076469421387, 0.9062116146087646, -0.07187094539403915, 1.4478720426559448, 0.9140624403953552, -0.3112011253833771, 1.1369729042053223, 2.023160457611084, -0.7147416472434998, 1.7714425325393677, 0.8478826284408569, -0.2978183329105377, -1.0046344995498657, 0.0014472748152911663, -0.6239947080612183, -0.4586522877216339, 0.5053150653839111, 0.43596547842025757, 0.9964885711669922, 1.3615400791168213, 0.46205005049705505, 0.9517470598220825, 0.1870097517967224, 0.17201285064220428, -0.41699397563934326, 0.7056394219398499, 1.109263300895691, 0.12741276621818542, -0.09059407562017441, 0.5318892002105713, 0.07632143795490265, -1.543511152267456, 0.7315276861190796, -0.8609671592712402, -0.09295240044593811, -0.035691872239112854, 0.4933934211730957, 1.9940446615219116, -0.12938357889652252, 1.4731241464614868, -1.8766964673995972, 1.711753249168396, -1.0842065811157227, 0.6130254864692688, 0.9844346046447754, -0.01839189976453781, 1.1980494260787964, 0.03193752467632294, -0.17868132889270782, -0.24293570220470428, 0.8095879554748535, 0.5002915859222412, -0.6060130000114441, -0.7298395037651062, 0.35759204626083374, -0.5091718435287476, -1.1964566707611084, -0.34442058205604553, -2.464733362197876, 0.25434932112693787, -0.2819909453392029, -0.6819632053375244, 0.32920166850090027, -1.3814536333084106, -1.5655477046966553, 0.3077507019042969, -0.635164201259613, 0.8075851202011108, 0.6422898769378662, -1.7708261013031006, -1.6124790906906128, -1.726122498512268, 0.27067193388938904, 0.14262787997722626, 1.1470811367034912, 1.3930681943893433, 1.1606123447418213, 0.6159738898277283, 1.0910630226135254, -0.6347408294677734, -0.92650306224823, -0.12207997590303421, 0.4925772249698639, 0.17429867386817932, -0.277680903673172, -0.2067444771528244, -1.448781967163086, 0.784338653087616, 1.4357837438583374, -1.6764984130859375, 0.9768640995025635, 0.3496938943862915, 0.9271314740180969, 0.045087751001119614, -0.2274409681558609, -0.2446511685848236, -1.058495283126831, -0.5084168314933777, -0.6836470365524292, 0.7501581907272339, -0.7650017142295837, 0.8838921189308167, -1.11822509765625, -0.21629446744918823, 0.090799979865551, -0.3736197352409363, -1.040825366973877, -0.5389297008514404, 0.6266612410545349, 0.7138957381248474, -0.08152258396148682, -1.1261649131774902, 0.3367667496204376, -0.4441656470298767, 0.6574153900146484, -1.1710435152053833, -0.5480295419692993, -1.0600804090499878, -1.5162245035171509, -0.9078124165534973, 0.21798697113990784, 0.4279765784740448, -0.25113645195961, -1.7195438146591187, -0.4655320346355438, -0.22761739790439606, -0.6431721448898315, -0.527407705783844, -0.27788418531417847, -2.0606377124786377, 0.08959464728832245, -0.13788414001464844, 0.12686052918434143, -0.6873252391815186, 0.45895761251449585, 0.04352346435189247, -1.0227115154266357, -0.2718125581741333, 1.5251712799072266, 1.767011046409607, 0.698566198348999, -3.037576198577881, -0.4193252921104431, 1.322847843170166, -0.14463257789611816, 0.6312451958656311, 0.9667749404907227, -1.3716158866882324, -0.14244413375854492, -1.5396621227264404, 0.40375417470932007, -0.9616983532905579, 1.5938529968261719, -1.2674334049224854, -0.950580894947052, -0.3823598027229309, 0.35019293427467346, -0.2780435383319855, 1.2208853960037231, 0.02811693772673607, -0.7015393376350403, 0.49807459115982056, 1.3429782390594482, 1.3155081272125244, 0.0691983699798584, -0.33807671070098877, -0.3864244818687439, -1.6164007186889648, -0.4890958070755005, -0.09581682831048965, 0.3323352336883545, -0.5031296014785767, -0.3023974895477295, -0.2989502251148224, 3.6446921825408936, 0.4570930600166321, -2.9250524044036865, 0.7387811541557312, -1.718475580215454, -0.04486081004142761, 0.7781182527542114, 1.2042075395584106, 1.6570353507995605, -0.7501877546310425, 1.172858715057373, -0.04237722232937813, -1.1633832454681396, -0.8718625903129578, 0.6556590795516968, -0.315845251083374, -0.42941731214523315, 0.948062002658844, -0.6674175262451172, -0.26870056986808777, 1.381170392036438, 0.051736053079366684, -0.12791621685028076, -1.4275152683258057, 0.06979258358478546, 1.3755117654800415, 0.7482786178588867, 0.8698175549507141, -1.8609130382537842, -0.8838626742362976, 0.9676283001899719, -0.5837128758430481, -0.15731066465377808, -0.4601964056491852, 0.024114759638905525, 0.6642472147941589, 0.9187791347503662, 1.5846493244171143, -0.35461917519569397, 0.6634421944618225, -0.24646781384944916, -0.4284968674182892, 1.9883450269699097, -0.843999445438385, -0.7966853976249695, -1.1435871124267578, -0.33680108189582825, 0.03518954664468765, 1.2850755453109741, 0.33855751156806946, -0.2885628044605255, 0.1655263751745224, -0.6477693915367126, 1.482971429824829, -1.143607258796692, 0.376185804605484, -0.3983841836452484, 0.25111261010169983, 0.0880758985877037, -0.9533869028091431, -1.0510482788085938, -1.1468478441238403, 0.033673036843538284, 1.0913742780685425, 0.014288893900811672, 0.9479700326919556, -0.7573590874671936, -0.2666266858577728, -1.8680405616760254, -1.964084267616272, -0.43405967950820923, -0.2864001393318176, -0.912689745426178, 0.9088643789291382, -1.2663549184799194, 0.02814679592847824, 0.9508782625198364, -1.469987154006958, -0.833320140838623, 1.2332249879837036, -0.36428454518318176, 1.1799116134643555, -1.407246470451355, 0.657060444355011, 2.9124813079833984, 0.8768143057823181, 0.5564545392990112, -0.02282382920384407, 0.020552534610033035, -1.0154870748519897, -0.9293230772018433, -0.9320828318595886, 1.813714623451233, 0.03858295828104019, 0.23230688273906708, 0.035509683191776276, 1.8291305303573608, -0.1444908231496811, -0.5618521571159363, 0.5750651359558105, 0.260492205619812, -0.8133096694946289, -0.6451870799064636, 1.9249769449234009, -0.21304166316986084, 2.1109912395477295, -0.24611228704452515, 0.8627707958221436, -1.1132431030273438, -0.30327025055885315, 1.4654464721679688, 2.081315517425537, -1.0810139179229736, 0.08672374486923218, -0.010626782663166523, -0.5633226633071899, -0.06550624221563339, 0.42136234045028687, -1.7288626432418823, -0.7800214886665344, 0.24984997510910034, -0.3838362991809845, 0.09103111922740936, 1.0734658241271973, 0.8490642309188843, 0.5498356223106384, -0.1208062469959259, -0.40004628896713257, 1.3586783409118652, -0.7768546938896179, -0.36700984835624695, -0.35822197794914246, 0.13446427881717682, 1.6181453466415405, -0.6224654912948608, 0.30516481399536133, 0.29639536142349243, -1.1368695497512817, 1.4641022682189941, 1.2668416500091553, 2.093221664428711, 0.17280961573123932, 1.384122610092163, -0.22572466731071472, -0.7247376441955566, -0.22595982253551483, -1.0073109865188599, 0.9020121097564697, 0.32161083817481995, 2.729203224182129, -0.040431827306747437, -0.13490618765354156, 0.7484370470046997, -0.12552779912948608, 0.0046597919426858425, -0.740118682384491, 0.5283104181289673, -0.21567708253860474, 1.0781539678573608, -0.4539215862751007, -0.7457907795906067, -0.5825454592704773, 0.881907045841217, -0.8766821026802063, -0.7615103721618652, 0.5528484582901001, 0.09040915966033936, 2.5173397064208984, -1.6845943927764893, -0.3851643204689026, -0.11526849120855331, -2.9215376377105713, -1.9446884393692017, 1.2613216638565063, 0.6161137819290161, 1.0109426975250244, 0.8827730417251587, -0.3601721227169037, 0.5690202116966248, 0.09006360173225403, 1.1182637214660645, -3.154623031616211, -0.09339425712823868, -0.5777048468589783, 0.17815092206001282, 0.11484472453594208, -1.9032760858535767, 0.8282529711723328, -0.5368058085441589, -0.018267512321472168, 0.713375449180603, 0.05285903811454773, -0.8686369061470032, -1.1094317436218262, 0.16969379782676697, 0.24457165598869324, 0.8401509523391724, 0.366000235080719, 0.4054242968559265, 0.814911425113678, 0.13175280392169952, -1.2038766145706177, 0.30259281396865845, -0.513666570186615, 0.5017743706703186, -0.1998816877603531, 3.934551477432251, 1.149269461631775, 0.5291216969490051, 0.10284195095300674, 1.718721628189087, -0.6622359752655029, -1.024290680885315, 1.3025709390640259, 0.38569554686546326, 1.0508815050125122, -1.924412488937378, 0.012358318082988262, 0.8665124177932739, -0.9196116328239441, 0.26308172941207886, -1.1291260719299316, -0.8520667552947998, 0.8980287909507751, 1.179653525352478, -0.23038256168365479, 0.42238864302635193, -0.11965534090995789, 0.8243410587310791, -1.3684128522872925, 0.07644843310117722, 0.811426043510437, 0.6966349482536316, -0.1449907422065735, -0.39470821619033813, -1.3099119663238525, -0.36153754591941833, -0.5043209195137024, -0.3404301404953003, -0.14998340606689453, -0.6223801970481873, -0.44109484553337097, -0.5028280019760132, 1.5800507068634033, 0.1574871838092804, -0.8421312570571899, -1.1079634428024292, 0.666915237903595, -0.3967066705226898, -1.3288087844848633, -0.5567107796669006, 0.6825101971626282, 2.518007278442383, -0.7562677264213562, 2.3520443439483643, 0.049243152141571045, 0.8902263045310974, -0.1619502454996109, -0.8908156752586365, 0.3724212944507599, 0.48856642842292786, -0.44798344373703003, -0.33545780181884766, -0.1686149686574936, 0.7088496685028076, -0.7599757313728333, -0.6629258990287781, -0.4840312898159027, 1.1534982919692993, 0.24778270721435547, 1.694451093673706, -0.31547901034355164, -0.1806481033563614, 0.6465415358543396, 0.35373687744140625, 2.11183500289917, -0.5126799941062927, -0.20365655422210693, 0.6651722192764282, 2.2614333629608154, 0.8353208899497986, 0.5133880972862244, -0.6709961891174316, 0.2949918806552887, -1.2376525402069092, -1.0714302062988281, -0.7276080250740051, -1.496660590171814, 0.15367765724658966, -0.0046180603094398975, -0.5283482074737549, 0.7195864319801331, -0.9145422577857971, -0.22455543279647827, -0.7167115807533264, -0.9320368766784668, -1.0219870805740356, 1.6083581447601318, -0.9439226388931274, -1.045443058013916, -0.23521628975868225, 1.6251959800720215, 2.2164292335510254, 0.8144382238388062, 1.1155955791473389, 1.5212016105651855, 0.2761367857456207, 0.05249778926372528, -0.14416831731796265, -0.7299260497093201, 0.5403105616569519, -0.1409890502691269, -0.20694421231746674, 0.38938426971435547, -0.035430505871772766, -0.19396425783634186, 2.5316965579986572, 1.3969879150390625, -1.5378559827804565, 0.700425386428833, 0.7831313610076904, -1.601407527923584, -0.023936741054058075, -1.6079360246658325, 0.80878084897995, -0.5548061728477478, 0.012309207580983639, 1.6076163053512573, -1.1700862646102905, -0.2182503044605255, 0.9461604952812195, -0.5208361744880676, -0.14615729451179504, -0.03988725319504738, 1.0624361038208008, 0.3599768877029419, 0.02632245421409607, 0.9419053196907043, 0.4204237461090088, 0.30141448974609375, 1.1127581596374512, 0.1229228600859642, 0.976402997970581, -0.16249746084213257, -1.3879705667495728, -0.5003933906555176, -0.5992364883422852, 0.09438837319612503, 0.42345303297042847, 1.2400068044662476, -1.2154769897460938, -0.1843676120042801, 1.7270383834838867, -1.0654847621917725, 0.5945488810539246, -0.8064885139465332, -1.2567713260650635, 0.17338994145393372, 0.12126048654317856, 1.3024916648864746, -0.21317504346370697, -1.2993963956832886, -0.06221770867705345, -0.125437393784523, -0.14919328689575195, -0.7845602631568909, 1.3252514600753784, -1.2099088430404663, 0.16090990602970123, -1.5428845882415771, 0.036829326301813126, 0.788347601890564, 1.9519562721252441, -1.1162745952606201, -1.1869401931762695, -0.7371863722801208, -0.2710586488246918, -0.3195462226867676, -0.30380961298942566, 0.5211426615715027, 0.5361230969429016, 0.3304325342178345, -0.5747815370559692, 0.13675564527511597, -0.5864992141723633, -1.7127147912979126, 1.1002541780471802, 0.32886797189712524, -0.29471975564956665, 0.057871103286743164, 1.4877166748046875, -1.4226717948913574, 0.33460456132888794, -0.07304450124502182, -0.7910286784172058, -1.3896735906600952, 1.5805612802505493, -1.8098137378692627, -0.9998387098312378, -0.36948639154434204, 0.11638007313013077, 0.20352542400360107, -0.4824941158294678, -0.4342598021030426, 0.7891241908073425, 2.5910630226135254, 0.6564554572105408, -0.08123257011175156, 1.1563652753829956, 1.5978305339813232, 0.014450824819505215, -0.40953585505485535, -1.004221796989441, -0.7893458008766174, 0.9275626540184021, 1.7344117164611816, 0.16566608846187592, -1.1356561183929443, 0.4867299199104309, 0.24881213903427124, -0.7339152097702026, 0.34218984842300415, -1.5348896980285645, -0.05321861803531647, -0.047151118516922, -0.25561967492103577, -0.41055959463119507, 0.43231043219566345, 1.6776820421218872, 0.4596928358078003, 1.0670934915542603, -1.8131392002105713, 0.451017826795578, 1.6773812770843506, 0.12212429195642471, 1.2224092483520508, -1.731229305267334, -1.4560126066207886, -0.6154181957244873, -0.9597717523574829, -0.3104263246059418, 2.134688377380371, 0.7812962532043457], [-1.0132209062576294, 0.7552175521850586, -0.1291821151971817, -0.8958715796470642, 0.12451421469449997, -0.08224219083786011, -0.9431100487709045, 2.199160575866699, 0.3161511719226837, 1.359451413154602, -0.567939817905426, -2.6656599044799805, 0.8215743899345398, 1.637444257736206, -0.43909817934036255, -1.0814131498336792, -0.8593382239341736, -0.012521423399448395, -1.7052383422851562, -1.6189113855361938, -1.7784109115600586, 0.8353250622749329, -1.8277887105941772, 0.5547007918357849, 0.8160556554794312, 1.2134898900985718, 1.386656403541565, -1.3466298580169678, 1.1633071899414062, -2.1890251636505127, -0.9864465594291687, -1.655172348022461, 0.2419448345899582, -0.47736242413520813, -0.686612069606781, -0.4122268259525299, -0.9413638114929199, 1.735860824584961, 1.1369231939315796, 0.7446036338806152, -0.262925922870636, -0.5655759572982788, 1.5388994216918945, 1.699223518371582, -1.0055128335952759, -0.425761342048645, 0.6783949732780457, 0.189222514629364, 2.041048765182495, 0.5974777936935425, -1.4088488817214966, -1.4707328081130981, 0.9395638704299927, 1.025888442993164, -1.0703243017196655, -0.5127593874931335, 1.7177315950393677, -0.7613714933395386, -0.2055027186870575, 0.9645389318466187, 1.4534854888916016, -0.3260921835899353, -1.8484671115875244, -0.340934157371521, 0.7646565437316895, 0.421249657869339, -0.17062066495418549, -0.05744094401597977, 0.16990718245506287, 0.9490588307380676, -0.27386274933815, -1.758734941482544, 0.14671017229557037, 0.7457348704338074, -0.8477883338928223, 0.42574986815452576, -0.7540813088417053, -0.1797455996274948, 2.1766180992126465, 1.4013280868530273, -0.073907271027565, -0.6114262938499451, -0.8170535564422607, 1.3151057958602905, -0.5877856016159058, -0.703121542930603, -2.416081190109253, -0.30263516306877136, -1.1780816316604614, -1.2911136150360107, -0.7848599553108215, -2.8997642993927, -1.538724422454834, -1.0829840898513794, 2.098327398300171, 1.378599762916565, -0.9155491590499878, 1.5780798196792603, -1.429758906364441, -0.026672331616282463, -1.690918207168579, 0.3825642764568329, 0.10225068777799606, 0.8244508504867554, -2.984570264816284, 0.4311818480491638, 0.31436222791671753, 0.6160866618156433, -0.5283033847808838, 0.23265546560287476, -0.25188693404197693, -0.29385313391685486, -1.3588109016418457, 0.3596852123737335, 0.8475971817970276, 1.2651190757751465, -0.09285151958465576, -0.7400845885276794, 0.08591043949127197, -0.9366670846939087, 0.8061065077781677, -0.17923229932785034, 0.23437418043613434, 0.7818772196769714, -0.4826148450374603, 1.0813883543014526, -0.32513147592544556, 0.80062335729599, -0.6815294027328491, -0.2708570957183838, 2.2753071784973145, 0.28356075286865234, 0.5316130518913269, -0.181711807847023, -0.3346000611782074, -1.1593660116195679, 0.046482499688863754, -0.822496771812439, -0.7399084568023682, 0.07832760363817215, 1.623529076576233, 2.481391191482544, 1.6121121644973755, 1.3876408338546753, -0.8408318161964417, 0.7752570509910583, -2.507537364959717, -1.015409231185913, -0.36526045203208923, -0.6415936350822449, 1.1574772596359253, 0.4675137400627136, -0.5963579416275024, 0.19926013052463531, -0.6524887681007385, -0.4412010610103607, 0.6394603252410889, 0.14957822859287262, 0.43537765741348267, 0.6165204644203186, -0.3387848138809204, 0.5408221483230591, -0.5561539530754089, -0.23089519143104553, -1.0806519985198975, 0.8455322980880737, -0.03998706489801407, 0.18763844668865204, -0.24554520845413208, -0.4916236996650696, 2.031939744949341, 0.011731816455721855, -0.9271432161331177, -1.1813783645629883, -0.7007148265838623, 1.2703570127487183, 0.36304667592048645, -0.5057953000068665, -0.9577674865722656, 0.5596867203712463, -0.6011120676994324, 0.25443923473358154, 1.4987581968307495, -0.02672465704381466, 0.8579591512680054, -1.1970858573913574, 0.8160213828086853, -0.5804553031921387, -0.6964182257652283, 0.008448072709143162, 1.545234203338623, 0.17072802782058716, 0.7632706761360168, -0.7021066546440125, -0.16857290267944336, 0.5625177025794983, -0.7502074837684631, 0.5646613240242004, 0.5543649792671204, 0.930531919002533, -0.06137440353631973, -0.7371337413787842, -0.3114006519317627, 2.2603442668914795, -1.2533457279205322, 1.2067365646362305, 0.9992929697036743, 0.14037981629371643, 0.15714794397354126, 0.3499608337879181, -0.9900645613670349, -0.30516862869262695, -1.066197156906128, -1.3087230920791626, -2.801393747329712, 1.1209182739257812, -0.6225825548171997, 0.15717338025569916, 0.04839378967881203, -0.30001723766326904, -1.4343897104263306, 1.059039831161499, 0.8349437117576599, -0.9867366552352905, 0.4189932644367218, -1.2344461679458618, -1.1985780000686646, 0.5891227126121521, 1.212924599647522, 0.08822259306907654, -0.33609941601753235, -1.0801312923431396, 0.02539166621863842, -0.27262693643569946, -1.4826945066452026, -1.6027460098266602, -0.6957630515098572, 1.4911402463912964, 0.45455658435821533, 0.014508000575006008, -0.6344776153564453, 0.14621412754058838, -1.8397198915481567, 0.23685921728610992, 0.25184205174446106, -0.5117167830467224, 0.6958202719688416, 1.1713454723358154, 0.15778915584087372, 0.29200777411460876, 0.9802176356315613, -1.075138807296753, -0.3184463679790497, 1.1412500143051147, 0.7421458959579468, 0.3714487552642822, 0.26904383301734924, 0.9065821766853333, 0.2958495616912842, 1.2559441328048706, 0.31269535422325134, 1.3245162963867188, 0.1569175124168396, 0.40448394417762756, 0.7732840180397034, 0.29136431217193604, -1.2243986129760742, 0.340546578168869, -0.48736944794654846, 0.8152293562889099, -0.11671687662601471, 0.7720993161201477, 0.6507595777511597, -1.2103136777877808, -0.013381858356297016, 0.16740380227565765, -0.23330232501029968, 0.49707701802253723, -0.14210058748722076, -0.5208423137664795, 1.670179843902588, -0.5088650584220886, -2.044541597366333, 0.40116962790489197, -0.8941444754600525, 1.003064513206482, -0.8750659823417664, 1.2638561725616455, -0.6430763006210327, 1.615723729133606, 0.20524482429027557, 0.36777669191360474, -1.3749297857284546, -0.31476029753685, -1.1497858762741089, -1.2235904932022095, -0.12707896530628204, -0.5567727088928223, -0.7225839495658875, -0.9006726741790771, -1.2203384637832642, -0.2948850691318512, -0.7267602682113647, -3.201756238937378, -1.2476239204406738, 1.3170348405838013, 0.9515738487243652, -1.1751919984817505, -0.283297061920166, -0.862708330154419, 2.4401819705963135, 1.4637621641159058, 0.7579370141029358, -0.9353636503219604, 0.996688187122345, -0.019508717581629753, 0.19808179140090942, 0.542242705821991, 1.4850165843963623, 0.9393382668495178, 0.3368373513221741, 0.24935556948184967, -1.7169021368026733, -0.8493372201919556, 0.47673118114471436, 0.35371527075767517, 0.518240213394165, -0.3499039113521576, 0.2855513393878937, 1.8286633491516113, 1.6442444324493408, -0.26197704672813416, -0.7161078453063965, -0.7974348068237305, 0.5885200500488281, -1.5194612741470337, -0.9447431564331055, -0.7259734272956848, -3.2392306327819824, -1.2193113565444946, 0.9171120524406433, -0.6668353080749512, 0.9217883348464966, 1.5371967554092407, 0.365930438041687, -0.5043165683746338, -0.05273072421550751, 2.0961575508117676, 1.6329548358917236, -0.1935688853263855, -0.6813966035842896, -1.7770662307739258, 0.3359062373638153, -1.459847331047058, -0.013472478836774826, -1.6031334400177002, -0.38041386008262634, 1.5729743242263794, -0.6831092834472656, -0.8317334651947021, -0.8216261267662048, -1.7541240453720093, 1.4805635213851929, -1.0659024715423584, -0.2291961908340454, -0.7864090800285339, -0.3730785846710205, 0.3451230823993683, 2.0585145950317383, -1.031691551208496, -2.1724979877471924, 0.5466268062591553, -1.3346565961837769, -0.011343264952301979, 1.0109326839447021, 2.0662240982055664, 1.1295559406280518, 0.7078263163566589, 1.9020459651947021, -0.7625932693481445, -1.0210342407226562, -0.06658930331468582, 0.3824756443500519, 0.7072378993034363, 1.2523393630981445, 0.0798390582203865, 0.6376238465309143, -1.1651884317398071, 0.5399007201194763, 1.424248218536377, 0.02132697030901909, 1.44778311252594, -0.813934326171875, -0.19179876148700714, 0.2857765853404999, 1.080788254737854, 0.8522041440010071, 0.04596579819917679, 0.21550561487674713, -0.92142653465271, -0.27147698402404785, 0.9078104496002197, 1.424178123474121, 0.3532398045063019, -0.6401829719543457, -1.7093617916107178, -0.2182658165693283, -0.6457219123840332, 0.1716000884771347, 0.9709908962249756, 0.7403486967086792, -0.4041856825351715, 1.0444835424423218, 0.23636023700237274, -0.12567763030529022, 0.8802242279052734, 1.3673629760742188, -0.30585208535194397, 0.4505474269390106, 0.5202863216400146, 0.33544424176216125, 0.9218643307685852, -0.7950780391693115, 0.40040647983551025, 0.3582129180431366, 0.18481363356113434, 0.3239535689353943, 0.8523849248886108, 0.12159411609172821, -0.3023964762687683, 0.25591903924942017, -0.0831318125128746, 1.086604356765747, -1.1144076585769653, 1.371532678604126, -2.9110515117645264, -1.9870750904083252, 0.16138947010040283, 0.15491093695163727, -1.0873428583145142, -1.0200799703598022, 1.047410488128662, -1.8348618745803833, 0.3975830674171448, 0.31028029322624207, 0.39976954460144043, -1.3696609735488892, 0.8854135870933533, 1.0118529796600342, -0.47579580545425415, -0.4741945266723633, -0.2073637694120407, -0.19936498999595642, 0.08623632788658142, 0.23060601949691772, 0.17394594848155975, 0.41675010323524475, -0.40725982189178467, 0.2786503732204437, 1.2754029035568237, -0.8703775405883789, 1.8035720586776733, -0.816728949546814, 0.21329200267791748, 0.31044450402259827, 0.10131499171257019, 1.6440058946609497, -0.3276195824146271, 0.388827919960022, 0.28606516122817993, -0.1887449324131012, -0.19747792184352875, -2.2782649993896484, 1.2998000383377075, -0.7496399283409119, 0.7684822082519531, -1.016296148300171, -0.09895151108503342, -0.35186538100242615, 0.4809732437133789, 2.220000743865967, -0.46600601077079773, -1.3737581968307495, -2.2738044261932373, -0.7797808051109314, 0.024477152153849602, -0.08059869706630707, -0.09143846482038498, -1.7766848802566528, -1.7637232542037964, 1.1023727655410767, 0.5368500351905823, 0.2717190682888031, 1.1128134727478027, 1.1522475481033325, 0.6569908857345581, 0.3561677634716034, 1.11203932762146, -0.138868510723114, 0.6065559983253479, -0.592472493648529, 0.5955151319503784, 0.38920989632606506, -0.3455963730812073, -1.099054217338562, 0.312298059463501, 0.3047289252281189, 0.21197551488876343, 0.9951944351196289, -1.1215109825134277, 1.6353607177734375, -0.7748088836669922, -0.1453934609889984, 1.31557035446167, -0.4930098354816437, 0.31318509578704834, 0.2281397432088852, 0.9076061248779297, -0.33264362812042236, -0.8735584020614624, 0.20103521645069122, 1.1984249353408813, -0.0913267657160759, -0.9923871755599976, -0.3104991316795349, -0.5083155035972595, -1.1815745830535889, 0.0588490329682827, -2.5842337608337402, -0.32224687933921814, -0.16261285543441772, 0.046808239072561264, 0.7440592050552368, 0.08019627630710602, 0.7804802656173706, 0.36978915333747864, 1.1033165454864502, -0.8280321359634399, -0.637795090675354, -0.30176517367362976, 0.5179041624069214, 0.12550225853919983, 0.2969340682029724, -1.2387019395828247, 0.48532870411872864, 0.42414790391921997, 0.7320058941841125, -0.0743953064084053, -1.5331873893737793, -0.4676603078842163, 1.4317665100097656, -0.5912037491798401, -0.023659009486436844, -0.13969914615154266, 0.6167687773704529, -0.05858604237437248, -0.5314205884933472, -0.798622190952301, 1.9762731790542603, 0.8110873699188232, 0.49722710251808167, 1.0370160341262817, -0.148556187748909, 2.097771644592285, -1.2809123992919922, 0.6353117227554321, -0.5404837727546692, -0.3573756217956543, -0.9654720425605774, 0.038335978984832764, 0.3735584616661072, 0.2479100525379181, 0.8309242129325867, -0.03589191660284996, 0.1681704819202423, 0.39708805084228516, -1.4003490209579468, -0.9082530736923218, -0.11048296838998795, 0.8025608658790588, 0.5041312575340271, -0.7084220051765442, 1.395228624343872, 0.18433377146720886, -0.5272446274757385, -1.4689198732376099, 0.31213149428367615, 1.9686094522476196, 0.10963843017816544, 1.05256986618042, -0.2929225265979767, -1.0589038133621216, 0.5742453336715698, 1.1088758707046509, 1.0857585668563843, -0.27363431453704834, 1.066443920135498, 0.5684865713119507, 0.1422869712114334, 0.07087364047765732, -0.25484421849250793, -0.9516621828079224, 0.7098106145858765, -1.2896968126296997, -1.153815746307373, -0.6539562344551086, 0.9826895594596863, -1.8444539308547974, -2.1608734130859375, -0.3871704339981079, -0.650395929813385, 0.3430527150630951, -0.47998595237731934, -0.4520449936389923, -1.7474188804626465, -0.32382306456565857, -0.9329906105995178, 0.41913285851478577, 0.12974345684051514, 0.4066257178783417, 1.2851470708847046, 0.40748322010040283, -0.1575341820716858, -0.26515331864356995, 0.10816610604524612, 0.09367845207452774, 0.615319550037384, -0.44072476029396057, -0.6860750317573547, 1.2184659242630005, 1.1592477560043335, 0.24462774395942688, -0.3620853126049042, 1.369688868522644, 1.2353719472885132, -0.6096733212471008, 0.4490398168563843, -0.02119850181043148, 0.9009283781051636, 0.26238176226615906, 0.8237141966819763, -0.4443790018558502, -1.2019819021224976, 1.0120084285736084, 0.4542295038700104, -1.5813486576080322, 0.6070093512535095, 1.7402328252792358, 1.572474479675293, 2.0663654804229736, 0.2750794589519501, 1.460520625114441, 1.7171968221664429, -2.0632028579711914, 1.080788254737854, 3.1063218116760254, 1.3230301141738892, -0.9516251683235168, -0.6752073168754578, 1.27485990524292, 0.49626123905181885, 0.1813938468694687, 0.29248228669166565, 0.11229438334703445, -1.1421698331832886, -0.6863881945610046, 0.5469115376472473, 0.03363800793886185, -0.028490208089351654, 0.20773924887180328, -0.653701663017273, 0.6333775520324707, -1.3501919507980347, 0.7088246941566467, 0.8206489086151123, 0.19610412418842316, -0.28356313705444336, 0.17117419838905334, -0.17597705125808716, -0.6001054644584656, -1.4985144138336182, 1.8286850452423096, -0.459816575050354, 0.7314193844795227, -1.438390851020813, -1.0883798599243164, 0.4890673756599426, -0.392987459897995, -0.833999514579773, -0.5560420155525208, -1.5297777652740479, -1.678878664970398, -2.156750202178955, -0.17934297025203705, 0.19262173771858215, -0.8516601324081421, 0.34434887766838074, 0.20338751375675201, -0.6714138984680176, -0.28768643736839294, 0.49410730600357056, 0.014002993702888489, 0.2687573730945587, -0.26424556970596313, 0.10999196022748947, -0.49863332509994507, -0.7562876343727112, -0.4857845604419708, 0.5033738017082214, -1.246955394744873, -1.453895092010498, 0.8045877814292908, 0.4476642310619354, 0.9360017776489258, -0.7799871563911438, 0.49446484446525574, -0.5154759287834167, -0.6130767464637756, 0.9112507700920105, -1.0337144136428833, -0.649533212184906, -2.4571444988250732, 0.5326863527297974, 0.08934514969587326, -1.0969310998916626, 0.6641209721565247, 0.7103075385093689, -1.6321176290512085, 1.4455856084823608, -2.2184693813323975, -0.7151982188224792, -0.019286731258034706, -0.25667503476142883, 0.0761328861117363, 0.033388275653123856, 0.2924541234970093, -1.2625300884246826, 1.1369582414627075, -1.6148923635482788, -0.9360738396644592, 1.2987638711929321, 0.8218672275543213, -1.5722222328186035, 0.4638292193412781, -0.5073373317718506, 2.885113000869751, -0.8140572905540466, -1.9497404098510742, -1.0791410207748413, 0.6934412121772766, -0.2029992640018463, 1.1891486644744873, 0.6301476955413818, 0.17938540875911713, 1.3095096349716187, -0.7085491418838501, 0.8951793313026428, -1.0424455404281616, 1.9223436117172241, 1.041182041168213, 0.842628538608551, 0.6289947032928467, 0.2890610992908478, 0.9224348068237305, 0.07680260390043259, 1.607470989227295, -0.0920516848564148, 1.2075711488723755, -0.027020515874028206, 0.8535796999931335, -0.16374628245830536, -1.5525532960891724, -2.0165016651153564, 0.8431142568588257, 1.901684284210205, 0.0797450840473175, -0.7924811840057373, 1.8549423217773438, 0.66936856508255, -0.8972341418266296, -0.6116437911987305, -1.2930727005004883, -1.5617880821228027, -0.897345244884491, 0.9492338299751282, 0.31479141116142273, 1.2716542482376099, 0.6688064932823181, -2.247338056564331, -0.9161699414253235, -1.6229801177978516, 1.3452702760696411, -1.383304238319397, -0.37342920899391174, 2.399125576019287, -0.14394856989383698, -0.33100906014442444, -0.8562915325164795, -1.2034521102905273, 0.5185469388961792, -1.676137089729309, 1.592780351638794, -0.18263183534145355, -0.5679933428764343, -0.5332953333854675, 0.7308225035667419, 0.9518882632255554, -0.7442905306816101, -1.4080766439437866, -0.40131139755249023, -0.38408681750297546, 1.228226900100708, -1.2244998216629028, 0.06166708841919899, -1.101282000541687, 0.3910329043865204, 0.14765894412994385, 1.0414917469024658, -0.5951911211013794, 0.08573823422193527, -2.2092063426971436, 0.6476978659629822, -0.7027162313461304, -0.2895248830318451, 0.3755752742290497, 1.1463613510131836, 0.07360223680734634, -0.8176313638687134, -0.2982218265533447, -0.5507586002349854, -0.8819867372512817, -0.03692071512341499, -1.0756120681762695, 0.8593173027038574, -0.34380903840065, -1.8235154151916504, -1.4388163089752197, 0.8921350836753845, 1.6129013299942017, -0.4352100193500519, 2.4469361305236816, 0.12620681524276733, -1.5332417488098145, -0.579852819442749, -0.4809825122356415, -2.1085546016693115, 0.010069279931485653, -0.43111276626586914, -0.08132417500019073, -0.582925021648407, -1.6398149728775024, -0.09730857610702515, 0.7858812212944031, 1.311403751373291, 0.8299882411956787, 0.43221041560173035, 0.4242473542690277, 0.5056776404380798, -0.8579949140548706, -0.14163121581077576, -0.9101037979125977, -0.5386506915092468, -0.021238232031464577, 1.1287492513656616, -1.360067367553711, -0.4698807895183563, 0.2798033356666565, 0.30181536078453064, 0.9787629842758179, 0.7030937671661377, 0.6531519293785095, -0.2797584533691406, -0.3380063772201538, 0.6145617365837097, 1.203749656677246, 0.36678534746170044, -0.6352851390838623, 1.562732458114624, 0.3332795798778534, 0.8313426971435547, -0.2760784327983856, 0.7317256927490234, -0.647854208946228, -1.9017685651779175, 0.00991920568048954, -1.7239209413528442, -0.21015486121177673, 0.384095698595047, -0.6326090097427368, -2.0111777782440186, 1.3167065382003784, -1.203013300895691, 0.07309698313474655, 0.8632103800773621, -0.6642816066741943, 0.6326271295547485, 0.3099340498447418, -0.39291539788246155, 0.9118240475654602, 0.8856870532035828, 0.06868792325258255, -0.6791565418243408, -0.656778872013092, 0.33231106400489807, -0.9500547647476196, 1.183161735534668, -1.6982687711715698, -0.13743096590042114, -1.1027685403823853, -0.2713048756122589, 1.1086688041687012, -1.950535774230957, -0.2887507379055023, -0.3830081820487976, -1.3509750366210938, 0.17763018608093262, 1.224541187286377, -1.1729828119277954, 2.3310391902923584, 0.20619599521160126, -0.18233370780944824, -2.4442532062530518, 2.06369948387146, 0.8249359726905823, 1.5031160116195679, 0.4889005720615387, -0.5958454012870789, 1.025016188621521, -0.5024833679199219, 0.15916186571121216, -0.6389890313148499, 0.038682714104652405, 1.2058931589126587, -0.34342724084854126, -0.8201010227203369, 0.04945463314652443, 1.7891508340835571, -1.5196501016616821, 0.5490143895149231, -0.5169567465782166, 0.00509036285802722, -0.9559028744697571, -2.1501104831695557, 0.8230780959129333, -1.1616637706756592, -0.7534045577049255, 0.06824874877929688, 1.7586179971694946, 0.3840624988079071, -0.727027177810669, 2.096806049346924, 0.17913001775741577, -2.3462417125701904, -0.3824881613254547, 1.1204863786697388, -0.1767820566892624, 1.2470390796661377, 1.1703031063079834, 0.40400126576423645, 1.9124729633331299, -0.657963216304779, 0.2640974521636963, 1.7113391160964966, -0.1759311407804489, -2.236598491668701, 0.06680155545473099, 0.043585870414972305, 0.5014827251434326, 0.34322094917297363, 1.3929944038391113, -2.1059718132019043, -0.5489752292633057, 0.40852051973342896, 0.8547658920288086, 1.1355903148651123, -0.6421611905097961, -1.9336334466934204, 0.6788645386695862, -0.9143427014350891, 0.018417609855532646, 1.935808777809143, 1.3895263671875, -1.7973588705062866, -0.85489422082901, -0.17908036708831787, 0.3049469590187073, -0.7998744249343872, 0.6653497815132141, 0.19213253259658813, -3.258532762527466, -2.2355239391326904, -0.2563527226448059, -0.3767763078212738, -0.5833737850189209, 0.1308852732181549, -0.04104834422469139, 0.31646689772605896, 0.3317863941192627, 1.1509426832199097, 0.5315758585929871, -2.8071117401123047, -0.8866617679595947, -0.4507150948047638, 0.10927750170230865, 0.18231837451457977, 0.5825687646865845, 1.6789369583129883, -0.9920150637626648, -1.122779130935669, -0.46653303503990173, -1.66525399684906, 0.5296633839607239, 0.16833096742630005, 0.9275014400482178, -1.1986746788024902, 0.43687885999679565, 0.07244051247835159, -0.02531448006629944, -0.9271206855773926, -1.3724638223648071, 0.4016662538051605, 0.6124646067619324, -0.7240695357322693, 1.1449387073516846], [0.3927478790283203, -1.3453798294067383, 0.4173874855041504, -1.6768314838409424, 1.2656264305114746, -0.5932035446166992, -0.6207759380340576, 1.2625638246536255, 0.43485358357429504, -0.3147488534450531, -2.136715888977051, 0.3987768292427063, 0.43981802463531494, 1.3297463655471802, 0.7697056531906128, 0.5058596730232239, -0.6827545166015625, 1.6093580722808838, 0.5190512537956238, -2.385566473007202, -0.2130567878484726, 0.6537178158760071, -1.5265296697616577, 0.26718994975090027, 1.4277533292770386, 0.5457786321640015, 0.01695280335843563, 0.5801573395729065, 1.549450397491455, -0.1073768138885498, -0.9127321243286133, 0.6654497981071472, -1.6539283990859985, 0.10459226369857788, 0.2437724769115448, 0.5776036381721497, -0.15217013657093048, 0.1974055916070938, 1.482766032218933, 0.8695318102836609, 1.7193652391433716, -1.076461672782898, 1.2702895402908325, -1.5056357383728027, 1.9037004709243774, 0.5331382155418396, -0.9469192028045654, -1.2186894416809082, -0.9779086112976074, -0.8276671767234802, 1.4365878105163574, 1.5033849477767944, -1.493485927581787, -0.8133571147918701, 0.06250395625829697, 2.7565901279449463, 1.4911302328109741, 1.2737360000610352, -0.7885076999664307, -1.0329676866531372, 1.2586312294006348, 0.2306794822216034, 0.8067556619644165, -0.8831691741943359, 0.015234705060720444, 0.8618528842926025, 1.7631402015686035, 1.020098090171814, 0.5096215009689331, -0.9515418410301208, -0.7347489595413208, -0.3192482888698578, -0.4913831651210785, -1.3372710943222046, 0.11205220222473145, -1.0931285619735718, -2.1927342414855957, -1.4520018100738525, 0.3244493901729584, 0.1362905651330948, -1.1980730295181274, -0.6646836400032043, 0.021637320518493652, 1.1673040390014648, -1.0559133291244507, 0.5162773728370667, -1.1691932678222656, 0.0622093603014946, -0.4351779520511627, -1.497044563293457, 1.0627464056015015, -0.3806719183921814, -0.8795884251594543, 0.09114304184913635, -1.0252009630203247, -0.9063901901245117, -0.20728567242622375, -1.4517529010772705, -0.01447975542396307, 0.9590936899185181, -0.23594117164611816, -0.4818616509437561, -0.9766718745231628, 1.2226579189300537, 0.6710289120674133, -0.9554231762886047, -2.0321950912475586, 0.5083940625190735, -2.063220977783203, 1.6201364994049072, 0.4713020324707031, -1.0985801219940186, 1.3335059881210327, -1.0652117729187012, -0.8506439924240112, -1.6388506889343262, 1.4123220443725586, 0.4464917480945587, 1.6141753196716309, 1.8806884288787842, 0.7624951601028442, 0.3675636351108551, 0.029457466676831245, -0.17429833114147186, 1.0050057172775269, 2.047930955886841, -0.5969429016113281, 0.28659746050834656, 1.7642314434051514, -0.6418429613113403, -0.7058461904525757, 0.16054196655750275, 0.656069278717041, -1.2296754121780396, 0.6729985475540161, -0.24862168729305267, 0.19480444490909576, 0.8427731394767761, 0.3695054054260254, 0.31184133887290955, 0.44957855343818665, -1.1689242124557495, 0.4084283411502838, 1.1219865083694458, -0.283040851354599, -1.1740853786468506, -0.7291249632835388, -0.1778039187192917, -0.8340573906898499, 0.27765953540802, -0.30414387583732605, 0.00902530923485756, -1.125855803489685, 1.2287276983261108, -1.0443458557128906, 0.6452471613883972, 1.9635651111602783, 2.784201145172119, -0.8775896430015564, -0.2917290925979614, 0.898768424987793, 0.6673199534416199, -0.5556107759475708, -0.7418295741081238, -3.3243188858032227, 0.012260562740266323, -0.4732327163219452, 0.11692758649587631, 0.834220290184021, 2.045661211013794, 1.1410160064697266, -0.16029727458953857, 1.636662483215332, 1.7610085010528564, 0.4928024113178253, 0.8219402432441711, -1.9154167175292969, -0.8463737368583679, 0.8777597546577454, -1.387365698814392, 0.929592490196228, 0.22617293894290924, -0.21253837645053864, -0.022315222769975662, 0.012752769514918327, -1.3670004606246948, 0.41707760095596313, -0.6169576644897461, 0.36371827125549316, 1.1428776979446411, 1.5806690454483032, 0.611397385597229, -1.036624550819397, -0.16617609560489655, -0.6766151785850525, 0.37007275223731995, -0.7927805781364441, 0.40725287795066833, 0.24860359728336334, -0.680698037147522, -1.50391685962677, -0.57587069272995, 0.20463994145393372, -0.2473699003458023, 0.8335713148117065, 0.23150034248828888, 2.155597686767578, -1.2818738222122192, -0.22898165881633759, -0.09629789739847183, -0.07938055694103241, 1.152161717414856, 0.4751080870628357, -0.6921530961990356, -0.8769163489341736, -0.8187938332557678, -1.1558674573898315, 1.323635220527649, 0.3353850245475769, 0.5173256993293762, -0.2061382383108139, 1.1215291023254395, -0.3540150225162506, -0.11893609166145325, 0.02977416291832924, 0.27591392397880554, -0.05680077522993088, -0.04183001071214676, 0.7338957190513611, -1.3111238479614258, -0.4476468563079834, -1.2180646657943726, -0.6974123120307922, -0.23047813773155212, -0.9324960708618164, 0.6619373559951782, 0.002197270281612873, -0.49086248874664307, -0.04046257212758064, 1.7387754917144775, -0.16057147085666656, 1.4997390508651733, -0.47789376974105835, 0.7804713845252991, 2.107355833053589, -0.5271340012550354, 1.1143054962158203, -0.08530216664075851, 2.328770875930786, -0.5233611464500427, -1.2788728475570679, -2.664428234100342, -0.5416428446769714, 0.2922796607017517, 0.8825846910476685, 0.8529627919197083, -0.5542817711830139, 0.624357283115387, 0.163802370429039, -0.1602938175201416, -0.8076987862586975, 0.40861520171165466, -2.8822102546691895, 0.8101604580879211, 1.1763644218444824, 0.6893593668937683, -0.42464208602905273, 0.9230227470397949, -1.056325912475586, -1.0681394338607788, 0.4872640371322632, -0.024809852242469788, -1.593870759010315, -0.2691854238510132, -0.6123217344284058, -1.1217392683029175, 0.6872963905334473, -0.46794939041137695, 1.5311541557312012, -0.38047245144844055, 0.03711880370974541, 2.786931276321411, 1.93894362449646, -0.6069870591163635, 0.10666859894990921, -0.2331269085407257, -0.12725701928138733, -0.9248661398887634, 0.5047783255577087, -1.3926752805709839, -0.8465050458908081, -0.06152341142296791, 0.054355304688215256, 0.6073417663574219, 0.2354125827550888, -0.2077692598104477, 0.8719605207443237, 0.0205546747893095, 2.5254106521606445, -0.6757732033729553, -0.5885436534881592, 0.9394577741622925, -0.5774518847465515, -2.172548770904541, -0.700126051902771, -0.03536944463849068, 0.35216954350471497, -0.14196161925792694, 0.5486414432525635, 0.6295483708381653, 0.2929368019104004, 0.9926662445068359, 0.08785227686166763, -0.46067413687705994, 1.08889901638031, 1.2257351875305176, 0.12893563508987427, -0.1445511281490326, 0.3808000087738037, 0.43323948979377747, -0.6731284260749817, 0.7812414169311523, -1.949365496635437, -0.27410805225372314, 0.7098563313484192, 0.573307991027832, 0.42734816670417786, 0.903060793876648, -0.2996313273906708, 0.08305301517248154, 1.5168381929397583, 0.00528496690094471, -2.2100324630737305, 0.4059704542160034, 0.0996343269944191, -0.578896164894104, -0.3161691725254059, -0.3094664514064789, 1.8544620275497437, 1.2442643642425537, -0.7971547245979309, 0.9309893250465393, -0.931186318397522, 0.4704011380672455, -0.5035796761512756, -0.9242147207260132, -0.04043355956673622, -1.146341323852539, 1.579099416732788, 0.2651934325695038, 0.7036842703819275, -0.4878123104572296, 0.5963757634162903, 0.8165745735168457, -1.2606545686721802, -0.47822484374046326, -0.818670928478241, 1.1588467359542847, 0.7963969707489014, -0.8936861157417297, 0.8520963788032532, 0.4900762736797333, -0.05078072100877762, 0.023381182923913002, -0.5630666017532349, 0.7352316379547119, -0.9367349743843079, -0.37093600630760193, 0.43481290340423584, 0.9279310703277588, -1.4462839365005493, -1.599814534187317, -0.6045511960983276, -1.575085997581482, -1.855554223060608, -0.9879617691040039, -0.16753549873828888, -0.5422104597091675, -2.0800108909606934, 1.299169659614563, -1.0384516716003418, 1.2948637008666992, 0.5629468560218811, 1.033599853515625, 0.02033338136970997, 0.930864155292511, 0.2991945743560791, 2.053086757659912, 1.830012559890747, 0.4830693006515503, -0.12167452275753021, 0.38418886065483093, -0.604758620262146, -0.5843707323074341, -0.4032522737979889, 0.4796973466873169, 0.3027614951133728, 0.34555482864379883, 0.9753177762031555, 0.8240598440170288, 0.7319349646568298, 0.9067769646644592, -0.6344975829124451, -1.219704508781433, 0.4151337146759033, 0.2163434475660324, -0.4689386487007141, -0.43483006954193115, -0.5205506086349487, 0.9341478943824768, 0.7259182929992676, 1.8652650117874146, -0.04436442255973816, 0.0011332286521792412, 1.1176148653030396, -0.14276748895645142, 0.24586808681488037, -0.18352527916431427, 0.15826110541820526, -1.2912888526916504, 1.199515461921692, -0.5758724212646484, 1.3034247159957886, -1.3649824857711792, 1.812337040901184, -1.3490601778030396, 0.16012901067733765, 0.17880216240882874, -0.5920014977455139, 1.295030117034912, -1.1341367959976196, -1.3938292264938354, -0.5549454092979431, -1.6209311485290527, -0.607367753982544, -0.5014585852622986, 0.6175572872161865, -1.9105119705200195, -1.0518434047698975, -0.39179593324661255, 0.8270037174224854, 0.05789029598236084, -0.23481440544128418, -0.3884301781654358, -1.1835410594940186, -0.5832951068878174, 0.08613552153110504, 1.1807507276535034, -1.5418033599853516, -0.35171735286712646, 1.9925593137741089, 1.3961430788040161, -0.6698386669158936, -0.09042849391698837, -0.703639566898346, 0.034127127379179, -1.632245659828186, -0.8567736148834229, 1.1836193799972534, -0.2651458978652954, -0.18459054827690125, 2.360686779022217, -2.378063440322876, 0.10582204908132553, 2.1779685020446777, 0.5782417058944702, 0.7202396392822266, 1.2774916887283325, 0.6725653409957886, 1.0866565704345703, 1.0249831676483154, -0.06238262355327606, 0.4936787784099579, -0.05424405261874199, -0.09196297824382782, -0.6967284679412842, -1.9742615222930908, 1.2304106950759888, 0.5085512399673462, -0.02179182879626751, -2.330272674560547, -0.4172004163265228, 0.9136465787887573, -0.42516571283340454, 0.15360473096370697, -0.24494656920433044, 0.07612579315900803, -1.689771294593811, 1.1339985132217407, 1.2368990182876587, -0.5911086797714233, 0.5757355690002441, -0.5911319851875305, -1.767871379852295, 0.904329776763916, -0.2574942708015442, -0.1559806913137436, 1.1076593399047852, 0.9619197845458984, -1.502798318862915, 0.06719204783439636, 0.14595001935958862, 1.2750537395477295, -1.5592812299728394, 0.8976815342903137, 0.01334631722420454, -0.6495151519775391, -0.8456974029541016, 1.1179661750793457, -0.390579491853714, 1.8498597145080566, 0.1460365653038025, -0.9772699475288391, -0.6907913684844971, -2.1038575172424316, -0.8135417699813843, 0.2602338492870331, -0.963036298751831, -1.3027012348175049, 0.31356191635131836, 2.0430502891540527, -0.6840243935585022, 0.4163477122783661, 1.4897314310073853, -1.6061697006225586, 0.5804832577705383, 1.0491807460784912, 0.7487722039222717, 0.2814983129501343, -1.1237006187438965, 0.1291474997997284, 0.34096723794937134, 0.17745448648929596, 1.3047935962677002, 0.9600088000297546, 1.0758806467056274, -1.5254920721054077, -1.361040711402893, -0.04738624393939972, -0.8739331364631653, 1.6727721691131592, 0.18042638897895813, -0.49119046330451965, 1.1985243558883667, 1.3898273706436157, -1.989116907119751, 0.986889660358429, 0.18197958171367645, 1.0187908411026, -0.0037868248764425516, 0.04488684609532356, -0.33024173974990845, 0.8398515582084656, -1.0102413892745972, -1.9427393674850464, 0.5661146640777588, 0.5648496150970459, 0.40578368306159973, 0.7723992466926575, 2.043058395385742, -1.3424650430679321, 1.571325659751892, 0.8572311401367188, -1.0056427717208862, -0.34645289182662964, -0.2475738227367401, 0.6639307737350464, -0.19525699317455292, -1.2247976064682007, -0.6818780303001404, -1.3259353637695312, 2.8672633171081543, 2.2872931957244873, -1.746057391166687, -0.4156167209148407, 2.3231916427612305, -0.6519923210144043, 0.661918580532074, -0.14452731609344482, -0.4017402231693268, 0.22249148786067963, -0.08803175389766693, 0.34859365224838257, 0.7311193346977234, -0.5707235932350159, -0.3003329932689667, 0.2992231249809265, 0.8179900646209717, 0.8316871523857117, 1.0790870189666748, -0.8277234435081482, -1.1316866874694824, 2.240581750869751, -1.476313829421997, -0.10577131807804108, -0.07074620574712753, -1.7396233081817627, -0.786970317363739, -2.774221897125244, -0.350461483001709, -0.0017608903581276536, -1.5498617887496948, 1.5103565454483032, -0.2930537164211273, -0.5837178230285645, 0.01821625977754593, -0.5915543437004089, -0.5204043984413147, -0.029546283185482025, 0.057588592171669006, 1.2183687686920166, 0.7321081757545471, -0.7713233232498169, 1.337627649307251, 0.24346761405467987, -1.0787845849990845, -0.9917477965354919, 1.976575255393982, -0.34535643458366394, -1.1894493103027344, 0.1208101361989975, -1.772754192352295, -0.23854583501815796, -1.376929521560669, 1.4280275106430054, 0.3967134356498718, 2.031158447265625, -0.4037429690361023, 0.07358549535274506, -0.8019298315048218, -0.9997216463088989, -1.7847492694854736, 1.1476486921310425, -1.127015233039856, -0.7474939823150635, 1.1628390550613403, 0.3402750790119171, 0.1765461415052414, -2.4944119453430176, -1.7599928379058838, 1.0647025108337402, -0.5352802276611328, -0.6292019486427307, -0.37173423171043396, -0.42915886640548706, -1.1418898105621338, -0.1671837568283081, 0.9373024106025696, -0.07226845622062683, 1.6684855222702026, 1.0720881223678589, -1.2197123765945435, 1.3585591316223145, -0.11380910873413086, -0.16468563675880432, -1.0673415660858154, 0.014368253760039806, -1.008817434310913, 0.5773834586143494, 1.083385705947876, 0.6528328061103821, 0.06612932682037354, 0.0031829611398279667, 1.5348460674285889, -0.9164777398109436, -0.06675712764263153, -0.42561137676239014, 2.087862968444824, -1.7512065172195435, 0.4781930148601532, -0.869935154914856, -1.770922303199768, 0.8827371001243591, 0.7252758145332336, 0.4123230278491974, 0.4386647343635559, 0.3896147310733795, -0.08721642941236496, 0.6318404078483582, -1.1565834283828735, -0.5406477451324463, -1.0172600746154785, -0.4203009605407715, 0.30779507756233215, 1.4800832271575928, 0.42083925008773804, 0.3220517337322235, 0.7827943563461304, -0.5937450528144836, 1.5082759857177734, -1.7101598978042603, 0.7115845680236816, 1.6900454759597778, -0.9644141793251038, -0.5221560001373291, -0.406777560710907, 0.8992669582366943, 0.11872033029794693, -0.5158913731575012, -0.4683755934238434, 0.697050154209137, 0.6028856039047241, -0.9311830997467041, 0.09829501807689667, 0.364636093378067, -0.17458103597164154, -0.6647772789001465, 1.8934823274612427, 1.006463885307312, 0.11881465464830399, 0.1953100562095642, -0.08437421917915344, -0.018170887604355812, -0.30442699790000916, 0.8954127430915833, -0.6395411491394043, -1.4518836736679077, -0.6296402812004089, -0.43567970395088196, 0.45583802461624146, -2.0107734203338623, -0.617730975151062, -0.9050171971321106, 0.44036781787872314, 1.6011213064193726, 0.36490681767463684, -0.5855662822723389, 0.9661641716957092, -0.35895830392837524, 0.8108317852020264, 0.12591509521007538, -0.4937511682510376, -0.26139065623283386, -0.1491706669330597, 0.12429801374673843, 0.7706191539764404, 1.8182390928268433, 0.705474317073822, 1.0847853422164917, -1.2463815212249756, 1.419415831565857, 0.941284716129303, 0.24117864668369293, -1.9075747728347778, 1.3826309442520142, -0.12208381295204163, -0.2551674544811249, -1.124614953994751, 0.8813501596450806, 0.487472265958786, 0.5261882543563843, 0.978115439414978, 0.918878436088562, 0.09528589993715286, 0.8878679275512695, 0.03647029027342796, 1.2348039150238037, -1.0934011936187744, -0.11541257798671722, -0.06170124560594559, 0.7918471097946167, 0.3307313919067383, 1.1626275777816772, 1.3680076599121094, -0.23572006821632385, 0.634953498840332, -0.2139052003622055, -0.8354300260543823, -0.9816811084747314, -0.7447432279586792, -0.5537378787994385, -1.8004621267318726, 0.7177032828330994, 0.6638755798339844, 0.31323713064193726, 1.5958046913146973, -0.3698417842388153, 0.6638234853744507, -0.34571921825408936, 0.8141300678253174, 1.9506882429122925, 0.9759074449539185, 0.7299544215202332, 1.7763806581497192, 0.8313679695129395, 1.184301733970642, 0.20072408020496368, -0.4613717496395111, -0.3375455141067505, 0.044837091118097305, 2.0142877101898193, 1.9830881357192993, -1.1926026344299316, 1.075881004333496, 0.3975074887275696, 0.2485233098268509, 0.22998486459255219, 1.4919604063034058, 0.619236171245575, 0.7170092463493347, 0.3881031274795532, -0.8769711852073669, -0.32782241702079773, -1.4862717390060425, -0.019708076491951942, 0.02842865325510502, 2.1059515476226807, 0.0888306051492691, -1.4032492637634277, 0.41898635029792786, -1.9269312620162964, 1.003163456916809, 0.4705524146556854, 0.3234414756298065, 1.8999780416488647, 0.29373472929000854, -0.07369132339954376, -0.2900562286376953, -1.628582239151001, -0.18339210748672485, 1.193707823753357, -0.2792380750179291, -0.10950782150030136, 0.5891922116279602, 0.02010936476290226, -0.35520651936531067, -0.7632107138633728, 1.8453220129013062, 0.6175253391265869, 2.0356051921844482, -0.23682451248168945, 0.6986730098724365, -2.032796859741211, 0.15406401455402374, 0.8027573227882385, -0.8825297951698303, -0.5654593706130981, -0.9147065877914429, 0.4878833591938019, -0.3993454873561859, 0.4055224657058716, -1.186850666999817, 0.4226342737674713, -0.5426097512245178, -0.7872498035430908, -0.4964044392108917, -2.349597215652466, 0.9299235343933105, 1.7499721050262451, 0.5532134175300598, 0.4415613114833832, -0.5546659231185913, 1.5520100593566895, -0.05322906747460365, -0.35782021284103394, -0.44406265020370483, 1.0755480527877808, 0.995400071144104, 2.7567594051361084, 0.026287168264389038, -0.07144061475992203, 0.5768586993217468, -2.218165636062622, 0.3500414192676544, -1.3556230068206787, -1.4454234838485718, 0.3789850175380707, -2.8823189735412598, -0.2785436809062958, -0.11870190501213074, 0.11905713379383087, 1.6741660833358765, -0.5537813901901245, -0.0211899783462286, 1.0309785604476929, -0.3724778890609741, -0.5118416547775269, 0.5221871733665466, 0.6484659910202026, 0.8237748146057129, -0.11098191887140274, 0.04857059568166733, -0.7702505588531494, -0.23975352942943573, -0.3002094626426697, 0.7436137795448303, 0.48595085740089417, 1.0735235214233398, -0.5018923282623291, -0.4794890284538269, 0.09242480993270874, 1.0532265901565552, 1.1979964971542358, 1.0913045406341553, -0.0637422502040863, 0.8046473860740662, -0.6733656525611877, -2.944066286087036, -1.295896291732788, -0.6646038889884949, -1.2976545095443726, 0.6216816306114197, -0.12650145590305328, 0.9787023067474365, 0.2863371968269348, 0.7090049386024475, 0.6024460196495056, -0.31756719946861267, 0.8573450446128845, -0.629500687122345, 0.9210227727890015, 0.13745537400245667, -1.0702201128005981, -0.5627202987670898, -0.5550338625907898, -0.38288602232933044, -0.10809692740440369, -0.3968242406845093, -0.5505871772766113, 0.9599552154541016, 0.5222221612930298, -1.6617909669876099, 0.724727988243103, 1.3444552421569824, 1.3592712879180908, 0.17245759069919586, 1.0075128078460693, -0.9096457362174988, -0.12674206495285034, -0.25658828020095825, 0.4985777735710144, -0.3767950236797333, -1.395039439201355, 2.0999093055725098, -0.4966559112071991, -0.6621723175048828, -2.2552413940429688, -0.47113311290740967, 0.8816583752632141, 0.2096783071756363, 1.0908452272415161, 0.03849560394883156, -1.2246499061584473, -0.5703315138816833, 0.13969279825687408, 0.08379799872636795, 0.9085312485694885, 0.40592455863952637, 0.9168693423271179, 0.2622951865196228, 1.1123095750808716, 1.4350552558898926, -2.1466832160949707, -0.8985199332237244, -1.5419297218322754, -0.4745435416698456, 1.4238814115524292, -1.538723111152649, 0.3997717797756195, 1.149049997329712, -0.3042283058166504, -0.16812214255332947, -2.1716196537017822, 0.09055609256029129, -0.23359757661819458, 1.3723692893981934, -0.8306025266647339, -1.1959376335144043, -1.278686285018921, 0.12621958553791046, -0.13018937408924103, 0.8779692649841309, -0.3462856709957123, -1.1442216634750366, 1.1401158571243286, 0.1318044811487198, -0.7158941626548767, -0.01697826199233532, 0.83897864818573, 2.1498918533325195, 2.77817964553833, 0.5760714411735535, 0.663524866104126, 0.10172707587480545, -1.4284796714782715, -0.6752379536628723, 0.39380958676338196, -1.1537549495697021, 0.4925324022769928, 0.6975803971290588, -0.24186117947101593, 0.3180413842201233, -0.06813537329435349, 0.26585280895233154, -0.8081415295600891, 2.469923257827759, -2.5147628784179688, 1.968587040901184, 0.45260825753211975, -0.4657479226589203, 1.1080961227416992, 0.9932476282119751, -1.2293788194656372, 0.18051934242248535, 0.7795488238334656, -0.010360736399888992, 1.3981287479400635, -1.0881381034851074, -0.349926620721817, -1.1620666980743408, -2.790393114089966, 0.7336493134498596, -0.31055116653442383, 1.148768663406372, -0.07247995585203171, 0.4543210566043854, 0.37933796644210815, 1.1043885946273804, -1.5128798484802246, 1.0417077541351318, 0.29508402943611145, -1.979591965675354, 2.1451609134674072, -1.406236171722412], [-0.59565269947052, 2.138894557952881, 1.233551263809204, 0.31314754486083984, 0.8589669466018677, 0.9627434015274048, 0.29685357213020325, -0.9205374121665955, 0.6136572360992432, -0.5286192893981934, -0.863193154335022, -0.64106684923172, 0.847364604473114, -0.008800975047051907, -1.3319275379180908, 0.16245955228805542, 0.5325759649276733, -0.6117954254150391, -1.756223440170288, 1.3507798910140991, -0.3134024739265442, 1.3083813190460205, -1.0953078269958496, -0.07207430154085159, 1.1306462287902832, 0.20304632186889648, -2.1277012825012207, -1.066819190979004, 0.9116016626358032, -2.4075162410736084, 1.3780640363693237, -0.34767934679985046, 0.1056581363081932, 1.559713363647461, -0.8536257147789001, 0.24624472856521606, 0.07274968922138214, 0.8747526407241821, -1.2404370307922363, -1.961922287940979, -1.1409220695495605, -0.857231616973877, -1.3925704956054688, -1.9356017112731934, -1.286805510520935, 1.1545321941375732, -0.40081554651260376, 0.15172870457172394, 0.8602628111839294, -0.035245321691036224, 0.2535994052886963, 0.8962810635566711, 0.2569481134414673, -0.5725951194763184, -0.6655205488204956, -0.14898189902305603, 1.074215292930603, 0.4795284569263458, -0.7952602505683899, 1.2721303701400757, -0.4737025201320648, 0.5237888693809509, -0.5521405935287476, -0.48181942105293274, -1.3294867277145386, 0.33852699398994446, -0.1265183687210083, 1.1941159963607788, -0.08251473307609558, -0.6748356223106384, 0.2948992848396301, -1.165217399597168, -1.8868716955184937, -0.8578778505325317, -0.2206636220216751, 0.6572979688644409, -1.0726816654205322, -1.453234076499939, -0.849009096622467, 0.8153244256973267, 0.7812528014183044, 1.7135027647018433, 0.49868690967559814, -0.10180234909057617, -2.8066983222961426, -1.5850173234939575, -0.5946123600006104, -0.06783932447433472, 0.09421868622303009, -0.9490942358970642, -0.8536947965621948, -0.38662219047546387, -0.06118332967162132, 1.0149363279342651, -0.9849874973297119, 0.5394529104232788, -0.18335846066474915, -1.2439494132995605, -1.6094763278961182, 1.2299392223358154, -1.0699236392974854, -0.38645222783088684, 0.05113651975989342, 0.22191329300403595, 1.1052203178405762, -0.1629800945520401, 1.0411927700042725, -1.8050131797790527, -0.5194398164749146, -1.0223835706710815, -0.5029910802841187, 0.4325522482395172, 0.048246148973703384, 1.3958252668380737, -1.5024471282958984, 0.7541213631629944, -0.9212371706962585, 0.34774935245513916, -0.2904030680656433, -1.3306242227554321, 0.9832834005355835, 1.0024266242980957, 1.1021314859390259, -0.44303593039512634, 0.04245986044406891, -0.40977978706359863, -1.1408840417861938, 0.6818596124649048, 0.017068875953555107, -0.3433401584625244, -0.3915329873561859, 0.22939260303974152, -0.16004090011119843, -1.3895461559295654, 0.8854030966758728, -0.24566347897052765, 0.10150818526744843, 0.8705337047576904, -0.4328405559062958, -0.04138685390353203, -2.0385031700134277, -0.6390262246131897, -0.2783060669898987, 1.2921366691589355, 0.4387000799179077, -0.13594786822795868, 0.06171976774930954, -2.254399299621582, 1.4145362377166748, 0.46102914214134216, -2.3674721717834473, -1.146302580833435, 0.9697790145874023, -1.7540974617004395, 0.39648932218551636, -0.9648841619491577, -0.0339326336979866, -0.17200574278831482, 0.6403587460517883, 0.8216388821601868, 0.15371142327785492, 0.6987993121147156, -0.6275960803031921, 1.7244913578033447, 0.3662683069705963, 0.5150691866874695, -0.23345646262168884, -0.7285411953926086, -2.059727668762207, -2.3702969551086426, 0.8170327544212341, -0.4281747043132782, -0.19761793315410614, -1.1961908340454102, -0.5598718523979187, 0.49720582365989685, 0.0769733265042305, 0.9626826643943787, 0.058957263827323914, -0.6676720976829529, 0.008831263519823551, -1.5310100317001343, -1.3648444414138794, 0.21284808218479156, -2.7431392669677734, -0.24261100590229034, -0.2738240361213684, -0.9031372666358948, 1.1246391534805298, 0.9976949691772461, -0.2287064641714096, 0.21574050188064575, -0.43569710850715637, 1.87242591381073, 2.26131272315979, -0.42792943120002747, 1.7221598625183105, 1.5209424495697021, 1.1731058359146118, -1.1539448499679565, 1.767008900642395, -1.197378396987915, 0.7092341780662537, -0.008583190850913525, 1.6208417415618896, -0.5210736989974976, -0.16774718463420868, 1.0529125928878784, 2.2196648120880127, -0.5045710206031799, 1.8436059951782227, 0.6485621929168701, 0.017282379791140556, -1.6697908639907837, 1.4629008769989014, 0.33895343542099, 0.1371779441833496, -0.9173791408538818, -0.21111860871315002, 1.0577890872955322, 1.0636818408966064, 2.0392603874206543, -0.4163084626197815, 0.906584620475769, 1.348871111869812, 0.8236911296844482, 1.9197359085083008, -0.4470401108264923, 1.2577054500579834, -0.20361323654651642, 0.10522081702947617, -1.2568902969360352, 0.5815693736076355, 0.361286461353302, 0.7084258198738098, -0.20328004658222198, -2.0315730571746826, 1.5573718547821045, -0.6783608198165894, -0.21459512412548065, -1.228804111480713, 0.1615150421857834, -0.667526364326477, -0.9118289351463318, -0.8734504580497742, -0.8561910390853882, -0.44305336475372314, 0.024202238768339157, -0.06079542636871338, -0.5693113207817078, -0.3165103495121002, -0.21775881946086884, -0.5254049897193909, 0.2515791654586792, -0.9569169878959656, 1.048904299736023, 0.5164240598678589, -1.2083337306976318, -0.6172265410423279, -0.035637035965919495, 0.8947625756263733, -0.716697633266449, -0.19625850021839142, -0.8840084075927734, -1.452538251876831, -1.1336561441421509, 0.8496689796447754, -1.5661853551864624, -0.005299598444253206, 0.8212825655937195, 0.15056312084197998, -0.19970090687274933, 0.053398072719573975, 0.32979699969291687, 0.3257974684238434, 0.48745062947273254, -1.6646438837051392, -1.667433738708496, -0.43847543001174927, -1.3006561994552612, 1.6709368228912354, 1.2397568225860596, 0.3392506241798401, 2.041592836380005, 0.10440569370985031, 0.22494757175445557, 0.4087148904800415, -0.4566032886505127, 0.6070756316184998, 1.3377267122268677, 0.6019884347915649, 0.704740583896637, -0.8143302202224731, 1.5901156663894653, -1.4642820358276367, 0.7582042813301086, 1.5650358200073242, 0.4747554361820221, -1.2391247749328613, 0.9785320162773132, 0.10818489640951157, -0.2756709158420563, -1.0975747108459473, 1.7440130710601807, -0.7725059986114502, 0.6903477907180786, 0.8118031024932861, -1.049055576324463, -1.4464514255523682, -0.3417010009288788, -0.6941933035850525, -0.9779998660087585, -0.5409361124038696, 0.045096516609191895, -0.29438772797584534, -0.6617941856384277, -0.3255327343940735, 1.261700987815857, 0.9904923439025879, 1.2714884281158447, -0.45482194423675537, -0.9006643891334534, 0.3136753737926483, 0.7896589636802673, 0.3749808967113495, -0.36565351486206055, 1.8308461904525757, -0.6088348031044006, 1.7207598686218262, -1.2592085599899292, 1.4622665643692017, 0.9552834033966064, 0.9526872634887695, -2.4152393341064453, 1.3261933326721191, 0.004561414010822773, -0.767480194568634, 1.0696412324905396, -0.6084669232368469, 0.18797676265239716, 0.6711794137954712, -0.19809268414974213, -0.9300698637962341, -0.17632892727851868, 0.09243698418140411, -1.0877022743225098, -1.2829300165176392, 0.9059166312217712, 0.6224320530891418, -0.4434807300567627, -0.045031215995550156, -0.3249325752258301, 0.8758869767189026, -0.2229035496711731, -0.548267126083374, -0.5780890583992004, -0.6969252824783325, -1.120036244392395, 0.507694661617279, 0.418495237827301, 0.2704697847366333, 0.789852499961853, 0.7142202258110046, 0.014948297291994095, 1.3753479719161987, -0.08118630200624466, -0.13575802743434906, -0.5176131129264832, -0.13614186644554138, 1.259169340133667, -1.067196249961853, 1.980938196182251, 0.513871967792511, 0.5205450057983398, -0.6358294486999512, 0.15554368495941162, 0.30675461888313293, 0.57684326171875, 0.21774697303771973, -0.8129410743713379, 0.1887824684381485, -2.0777199268341064, -1.103511929512024, -0.2982710301876068, 0.2202053666114807, -0.5745772123336792, 0.17968586087226868, -0.028716037049889565, -0.07054096460342407, 0.9114537835121155, -0.04436910152435303, 0.011616825126111507, -1.9865320920944214, -0.12316428124904633, 0.8388853669166565, 0.1507912576198578, -2.4006903171539307, 2.0493478775024414, 0.4438340961933136, -0.825860857963562, 0.3867882788181305, 0.11760114878416061, -0.7500923871994019, -1.7192124128341675, 0.37700414657592773, -0.20171168446540833, -0.3668533265590668, 0.8039317727088928, -1.0065507888793945, 0.2848161458969116, 1.2686010599136353, 1.4821815490722656, -0.5699344873428345, 0.30772680044174194, -0.1344844102859497, 0.32549744844436646, -0.5348588228225708, 1.0811152458190918, -0.42331406474113464, -0.8299784064292908, -0.31437429785728455, 0.5338287353515625, -1.1457542181015015, 0.6210758090019226, 1.5449107885360718, 0.2473902851343155, 0.7019137740135193, 0.14376989006996155, -1.4172966480255127, -0.08579885214567184, -1.1193103790283203, 0.03982386738061905, -0.2596687078475952, -1.5629174709320068, 0.08829161524772644, -1.09605872631073, 1.8967392444610596, 0.7873492240905762, -0.9070172905921936, 0.04721122980117798, 0.9340753555297852, 0.64812171459198, 0.11222520470619202, -2.6538968086242676, 0.41867342591285706, -1.180448293685913, -1.1621874570846558, 2.7559187412261963, 1.4724756479263306, -0.2303018867969513, -0.40102481842041016, -1.9179655313491821, -1.0483342409133911, 0.8719692230224609, -0.3417853116989136, -0.6831780076026917, -0.6066408157348633, 0.6939243674278259, 0.9359591007232666, -0.5323526263237, 1.2172030210494995, 0.09343447536230087, -0.6824162006378174, -0.15231655538082123, -0.7389267086982727, -0.7863293886184692, -0.9613775014877319, 1.055328369140625, 0.7197181582450867, -0.9674577713012695, 0.21590067446231842, 0.881921112537384, -1.5761302709579468, -0.7424020767211914, -0.3067527115345001, 1.0665594339370728, -0.7173977494239807, 0.13278062641620636, -0.6149317026138306, -1.1606720685958862, -0.6773933172225952, 0.981299877166748, -0.3295128047466278, 0.3708682358264923, -0.4455401301383972, 0.860602080821991, 1.338602900505066, 1.323649287223816, -0.1399284303188324, 1.5758247375488281, 0.8565264344215393, 0.03563104569911957, -1.344222903251648, -0.24403701722621918, 0.12957409024238586, -0.4740731716156006, -0.6922639608383179, 1.6891374588012695, 0.5503991842269897, -0.14414075016975403, -0.3603431284427643, -0.05586332082748413, 0.3230518698692322, -0.39639538526535034, -0.8726937174797058, -0.13209640979766846, 1.0216642618179321, -1.9332468509674072, 0.20260101556777954, 0.8932403326034546, -0.05188759043812752, -0.6199408769607544, -0.3583822250366211, 0.7110128998756409, -1.860331654548645, -0.3147448003292084, -1.032723069190979, -1.5698922872543335, 0.4413599967956543, -0.6123849749565125, 0.29448339343070984, 0.948656439781189, -1.26520574092865, -0.9629493951797485, -0.7988353371620178, -0.9940198063850403, 0.07987922430038452, 0.49364393949508667, -0.5023559927940369, 0.8179358839988708, -1.5543352365493774, 0.1271606683731079, 0.7123691439628601, -2.784627914428711, -0.020458748564124107, 1.7499909400939941, -0.5135654211044312, 0.199017733335495, -0.8563215732574463, 1.1620583534240723, -0.3509811460971832, -0.4500901699066162, 1.1919103860855103, -1.900098443031311, -1.5919655561447144, -1.0020009279251099, 1.0278712511062622, 1.206824779510498, 1.2759097814559937, 1.0062811374664307, -1.123616099357605, -1.2869431972503662, -0.13265222311019897, 0.02299228124320507, -1.8766357898712158, -0.618148922920227, 0.3324790298938751, 1.5076580047607422, 0.06978840380907059, 0.2818969786167145, -1.172895073890686, 2.4557783603668213, -2.2097301483154297, -0.6014569997787476, 0.04724392294883728, 0.26934438943862915, -0.9226891398429871, -1.3436745405197144, 0.25509777665138245, -0.2620853781700134, -0.04790365323424339, 2.2588164806365967, -2.229020118713379, 2.792712926864624, -0.6447264552116394, 0.14031995832920074, -0.3912547826766968, -0.6529240012168884, 1.3412883281707764, -0.3938139081001282, -0.6059619784355164, -0.269930362701416, -3.438739776611328, 1.6398577690124512, -1.8491120338439941, -1.2016831636428833, -1.7424087524414062, 1.6656663417816162, 1.0527187585830688, -0.9241957068443298, 0.09994076937437057, 0.7253555059432983, -1.043593168258667, -0.07638854533433914, -1.4422988891601562, 1.4243804216384888, -1.1674288511276245, -0.690192461013794, 0.3137340545654297, -0.8754603862762451, -1.5623795986175537, 0.37858477234840393, -0.669157862663269, -0.29884347319602966, 1.067992925643921, -1.062320351600647, 1.2358702421188354, -0.2674553394317627, -2.877377510070801, -0.9064229130744934, -0.7536240220069885, 0.7390104532241821, -1.395836591720581, 0.09075325727462769, 1.0279541015625, 0.149529367685318, -0.7827773690223694, -0.23665806651115417, 1.6611533164978027, 1.5297462940216064, 0.9933831095695496, -1.4144724607467651, 1.6971410512924194, -0.025047799572348595, -0.1615453064441681, -0.23042930662631989, 1.2787275314331055, 2.0362777709960938, 0.34470629692077637, 0.5116208791732788, 1.5697640180587769, -0.537501335144043, 0.23638997972011566, -0.7728015780448914, 1.3454495668411255, -0.11403295397758484, 1.8392435312271118, 0.7233010530471802, 0.8832194209098816, -0.4181436598300934, -1.7290772199630737, 0.5586559176445007, -0.7739917635917664, 0.17594240605831146, -0.6693182587623596, 1.2177375555038452, 1.188197135925293, 0.01478576846420765, -0.9458336234092712, -0.7497154474258423, -2.2828516960144043, -0.6196022033691406, 1.073111891746521, 0.44061699509620667, 1.1727999448776245, -0.5318235158920288, 0.8269563913345337, -0.09350447356700897, 0.14315399527549744, -1.7941468954086304, -0.6131806969642639, -1.0098053216934204, 0.251849889755249, -2.9329142570495605, 0.73702073097229, -1.0159662961959839, 0.208334818482399, -1.3208457231521606, -1.0757076740264893, -0.12231551855802536, -0.20792916417121887, 1.01682710647583, -1.0339844226837158, 0.14327377080917358, -0.30192187428474426, -0.08871489018201828, -0.14269351959228516, -1.2197766304016113, -0.5003520846366882, 0.05027923732995987, -0.4669770896434784, -0.4001002013683319, 1.1774275302886963, -0.5894289612770081, -1.4910355806350708, -0.809839129447937, -0.14564663171768188, -0.1731618344783783, 0.7185639142990112, 2.545180320739746, -0.8472278118133545, 0.41505077481269836, 0.5766224265098572, -0.017051320523023605, 1.2492871284484863, -0.7250152230262756, 0.4313305616378784, 0.5337045788764954, -1.6075105667114258, 1.1958677768707275, 0.3723668158054352, -0.37735360860824585, -1.2425494194030762, -0.7201346158981323, -0.7584802508354187, -0.5666298866271973, 0.32912254333496094, -0.6742688417434692, 0.21841342747211456, -0.8868544101715088, 0.7711848616600037, -0.4000340402126312, -0.10292619466781616, -1.3318305015563965, 1.1918100118637085, 1.1959681510925293, -0.28550058603286743, 0.24755242466926575, -0.6289473176002502, 0.2148326337337494, -0.7710755467414856, 0.288555383682251, -1.1172760725021362, -0.5725887417793274, -0.9916217923164368, 1.675044059753418, 0.8308600187301636, 0.7574109435081482, 0.6029897332191467, -0.21437785029411316, -0.754758894443512, -0.495757520198822, -1.3228321075439453, -0.24942435324192047, -0.9518234133720398, 1.4433319568634033, -0.051036130636930466, -1.2706542015075684, 0.8244897723197937, -0.012654307298362255, 0.17722231149673462, 0.35188958048820496, -0.3309600055217743, -0.6124549508094788, 0.3718021810054779, 0.9985791444778442, 0.5557836294174194, 1.0705808401107788, 1.5561811923980713, 1.303865909576416, -1.6521415710449219, -0.7895106077194214, -1.276277780532837, -0.16454359889030457, -1.4827648401260376, -1.7511398792266846, -1.86699640750885, 0.16469204425811768, -0.1549779176712036, -1.4309600591659546, 0.13844023644924164, -0.9923547506332397, 0.7013097405433655, -0.21203386783599854, -0.22861997783184052, -0.07795535027980804, 0.6711075305938721, 0.3491518795490265, -1.2362130880355835, 0.4512331187725067, 0.16858699917793274, 1.0967581272125244, -1.8664963245391846, 0.036393750458955765, -0.2793402671813965, 1.4791942834854126, 0.610750138759613, 0.4459085762500763, -1.0397840738296509, -0.5288273096084595, 0.8499610424041748, 0.5049866437911987, 0.5442022681236267, -0.3136935532093048, -0.7201399207115173, -1.0807801485061646, 0.10205638408660889, 0.16495618224143982, 1.3521251678466797, 0.29774269461631775, -0.5405998826026917, -0.4232991933822632, -0.8804948329925537, -1.4875568151474, -1.049881935119629, -1.5180166959762573, -0.6786162853240967, 1.8255895376205444, 0.34784215688705444, 0.21308237314224243, -1.2525968551635742, 1.1139366626739502, -0.17796102166175842, -1.547505259513855, -1.3131886720657349, -0.5326405167579651, 3.2523014545440674, 0.07203041017055511, 0.811285138130188, -0.43748095631599426, -0.09283721446990967, 0.14454127848148346, 0.3385389447212219, -0.055398356169462204, -0.03629006817936897, -0.07151991128921509, 0.14223028719425201, -0.7688581347465515, 0.7839349508285522, -1.4322844743728638, -0.022734511643648148, 0.4381679594516754, -1.179277777671814, -0.044407330453395844, -0.16501614451408386, 0.6776667237281799, 1.0908855199813843, 0.06559805572032928, -0.2335609346628189, 0.196064755320549, -1.4752033948898315, -0.7385969758033752, -1.9979333877563477, -0.117037832736969, 0.7761955261230469, -0.03894060105085373, 1.678977608680725, -0.5756992101669312, -0.03238551691174507, -0.6249132752418518, -0.11607082188129425, -1.1264806985855103, 1.322201132774353, -1.6322753429412842, -0.028861029073596, -0.5358285307884216, -0.7939375042915344, 0.6690424680709839, 0.4733152687549591, -0.2392558455467224, 1.445979356765747, -0.9456596374511719, 0.4787071645259857, 1.3698855638504028, -0.033639319241046906, 1.776329517364502, -0.7056885957717896, -0.2829820215702057, 0.058171678334474564, 0.9195632934570312, -0.6382035613059998, -0.03996005654335022, -0.36535096168518066, -0.019612658768892288, -1.1502227783203125, 1.7191673517227173, -1.0074877738952637, 2.221475124359131, -1.0083301067352295, -0.9217733144760132, 0.9121423363685608, 0.7986321449279785, 1.1617770195007324, 0.37925297021865845, 1.8744745254516602, -1.9649957418441772, 1.6069868803024292, 0.3876798152923584, 0.7653744220733643, -1.277739405632019, -1.1767199039459229, 2.8813419342041016, -0.8026956915855408, 0.1370275318622589, -0.03584687411785126, 0.9684820175170898, -1.1076616048812866, -1.7439696788787842, -0.8768942952156067, 1.6975089311599731, -0.11110328882932663, 0.8434100151062012, 0.30161502957344055, 0.48044508695602417, 0.598825991153717, -0.4059006869792938, -0.9402757883071899, -1.2975975275039673, -0.23214191198349, -0.034725937992334366, 0.6158071160316467, 0.5955167412757874, -0.8234912157058716, -0.49094149470329285, 0.46593716740608215, 0.7726407647132874, 1.0180466175079346, 2.0252246856689453, 1.0609790086746216, 1.5772559642791748, 1.2281765937805176, 0.013725892640650272, 1.2474020719528198, -0.07088911533355713, 1.2830966711044312, 0.7276329398155212, -0.2115703821182251, -1.1206259727478027, 0.8235387206077576, -1.5818977355957031, 0.3193497359752655, -0.049285806715488434, -0.6549279689788818, -1.4574780464172363, -0.9378523230552673, 0.17603939771652222, 1.2776299715042114, -0.16875778138637543, 0.7694109678268433, 1.041290044784546, -0.22099274396896362, 0.23165157437324524, 0.4267576336860657, -2.1746938228607178, 0.598221480846405, 0.9963575601577759, -0.15812329947948456, 0.23735179007053375, 0.2556595206260681, 0.3527238070964813, -1.322202444076538, 0.8936326503753662, 0.7405433654785156, -0.881600558757782, -0.35013628005981445, -0.8487809896469116, 1.6362484693527222, 0.8451789021492004, -1.8221014738082886, -0.7436477541923523, 1.2798819541931152, -0.567436695098877, 0.24811474978923798, -0.6601988673210144, -1.0843013525009155, -0.16908477246761322, -1.693808913230896, 2.6469311714172363, -0.12298551201820374, 0.3625221252441406, -2.4155197143554688, -0.5790714025497437, 1.3134503364562988, -0.7913735508918762, -0.39008572697639465, -0.8137418031692505, 1.0122493505477905, -0.37302884459495544, -0.8408545255661011, 0.87706059217453, -0.7921746373176575, 0.1647268384695053, -0.25091949105262756, 1.2848827838897705, -2.289984941482544, 1.2655702829360962, -0.28070157766342163, -0.7407284379005432, 1.1055692434310913, -0.06564079225063324, -0.5570849776268005, 0.4004565179347992, -0.09462083131074905, 0.5542685985565186, -0.1433095484972, 1.6960501670837402, 0.1567709743976593, 1.1780891418457031, 0.4573798179626465, -0.2822939157485962, 0.15295428037643433, 1.0885719060897827, -2.1017677783966064, -0.29783177375793457, 0.5106492042541504, 0.6676700711250305, 2.01273775100708, 1.174530267715454, -0.5044559240341187, 0.23307543992996216, 0.5720966458320618, -0.8446888327598572, -1.1816792488098145, -0.47322624921798706, -0.9256575703620911, -1.5320113897323608, -0.7772448658943176, -2.615990161895752, -0.668765127658844, 0.38494136929512024, -1.2905305624008179, -0.49804171919822693, -0.5522054433822632, 0.7909322381019592, 0.19903816282749176, -1.11823308467865, -0.5102002024650574, 0.730069637298584, 0.46245282888412476, 1.2643952369689941, -0.029260525479912758, 0.5385739803314209, -1.3401163816452026, 0.024834759533405304], [-0.3957318365573883, 0.01934676058590412, -0.3497082591056824, -0.8404890298843384, -0.39889535307884216, -0.014271631836891174, 0.1930444985628128, -0.00631299102678895, 0.42825832962989807, -2.031423568725586, 1.1509389877319336, 0.9620122313499451, 1.262040615081787, 0.31115487217903137, 0.170027494430542, 0.42976030707359314, 2.087057113647461, 0.6835604906082153, -0.6803677678108215, 0.18738733232021332, 1.2784863710403442, -1.2939616441726685, -0.9465365409851074, -0.73094642162323, -1.967317819595337, -0.209575355052948, 0.44984161853790283, -0.3118039071559906, -0.6628146767616272, -0.1837756633758545, -0.37843480706214905, 0.4642515778541565, 0.25513654947280884, 0.5783078670501709, -0.06383112818002701, -0.9134830236434937, 0.8160778880119324, -0.26578330993652344, -0.8559421300888062, 1.437394618988037, -1.180497169494629, 0.04216854274272919, -0.4579293131828308, -1.0199720859527588, 0.14686594903469086, 1.9465519189834595, -0.2552676498889923, -0.34280160069465637, -1.09197199344635, 0.624153196811676, 0.8187550902366638, -0.2715877592563629, -1.5927565097808838, -0.1620463728904724, 1.1843079328536987, -1.0280368328094482, -1.0836198329925537, 1.3288724422454834, -0.8133224248886108, 0.6034471988677979, 0.6124608516693115, 1.505537986755371, 1.5101017951965332, 0.3189525306224823, -2.2683115005493164, 0.2996293008327484, -0.7358382940292358, 0.6283884644508362, -1.0892630815505981, 0.30750641226768494, 0.7417899370193481, 0.0305397417396307, 1.1455010175704956, 0.32659411430358887, -1.6601191759109497, -0.570101797580719, 0.8362326622009277, -0.8374567031860352, 1.7021620273590088, 0.26920807361602783, -0.055142592638731, 0.1223439872264862, 0.35180801153182983, 0.20555652678012848, 0.5725215673446655, -1.3452227115631104, 0.9106150269508362, 0.35480818152427673, 1.1612427234649658, -1.9022313356399536, -1.0559085607528687, -0.3632190227508545, -0.32496097683906555, 0.6540993452072144, -0.43351981043815613, -1.233564853668213, -0.9559987187385559, 0.26973384618759155, -0.9249889254570007, -0.5024248361587524, 1.8130302429199219, 1.8031458854675293, 0.05071703717112541, -0.6014760732650757, 1.6799367666244507, 3.009010076522827, 0.43333014845848083, 0.2144334316253662, 1.7112194299697876, -1.2401236295700073, 0.7772383093833923, -0.35267579555511475, -0.10366763174533844, 0.5509699583053589, 0.4638344347476959, -0.5979282855987549, 0.3772389888763428, 1.239174723625183, 0.219646617770195, 0.8631170988082886, 0.8986093401908875, 0.4080042839050293, -0.5760253071784973, 1.6569181680679321, -0.37142467498779297, -0.19695475697517395, -0.48140376806259155, 1.311374545097351, -1.1340107917785645, 1.6928831338882446, 0.44341611862182617, -1.4451731443405151, 0.1905849277973175, 1.8243556022644043, -0.7767031788825989, -1.019831895828247, 0.9740419387817383, -0.20527900755405426, 1.2214034795761108, 0.017911437898874283, 0.0027386751025915146, -0.2623037099838257, -0.5854825973510742, -0.20664022862911224, 0.23670260608196259, 0.3431370258331299, -0.5495514273643494, -1.3061175346374512, 0.6833646893501282, -0.632307767868042, -0.14841879904270172, 0.7227997779846191, 1.4371479749679565, -1.082364559173584, -0.22234803438186646, 0.0718347355723381, -0.20852647721767426, -0.264342725276947, 0.7319924235343933, 0.4888099133968353, -0.9533329606056213, -0.6697255373001099, 0.5928090810775757, 1.3668718338012695, 0.38617798686027527, 0.7206889986991882, -0.2680710554122925, -0.9262620210647583, -2.038745403289795, 0.4279923439025879, 0.2553517818450928, -1.3936479091644287, 0.8254175782203674, -0.6209410429000854, -1.934499979019165, -1.4596645832061768, 0.3521198332309723, 0.7244875431060791, 0.9149731397628784, 0.01554484385997057, 0.6891594529151917, -0.23113194108009338, -0.8934730291366577, -1.8883787393569946, 0.021756909787654877, 1.1281393766403198, 1.0136687755584717, -0.18800577521324158, -0.9373065233230591, 0.780786395072937, 0.18072371184825897, -1.3253693580627441, -0.3267771601676941, -1.6446001529693604, 1.0164793729782104, 1.8733580112457275, -0.14754287898540497, -0.4388381838798523, 0.4077749252319336, -0.026110416278243065, -1.025956630706787, -0.05099952965974808, -1.9144333600997925, -1.451236605644226, -1.0204787254333496, 2.761789560317993, 0.6201616525650024, 0.2879924178123474, -0.7116250395774841, 0.40147674083709717, -0.0996001809835434, -1.7349319458007812, -1.0349218845367432, 0.07486985623836517, -0.291348397731781, 1.5556954145431519, -0.10183650255203247, -0.2666891813278198, -0.24151331186294556, -0.24205228686332703, 1.099452018737793, 0.7413270473480225, 2.4898641109466553, -0.15834859013557434, 1.1201071739196777, 0.08668417483568192, 1.0179939270019531, 0.09862218797206879, 1.3249536752700806, 0.015108241699635983, -1.230719804763794, 0.3150891959667206, -0.6027857661247253, 1.1069072484970093, -1.0215134620666504, -0.5415439605712891, 0.7777169942855835, 0.2748688757419586, 1.4494999647140503, 0.6314118504524231, -0.38431185483932495, 2.099177598953247, -0.9110041260719299, 0.6096863150596619, 0.9783507585525513, 0.8789311051368713, 0.22084960341453552, 2.923447608947754, 2.3786025047302246, -1.0462793111801147, -1.3583322763442993, -0.2103458046913147, -1.0136256217956543, -0.12963137030601501, -0.6435852646827698, 1.9195556640625, -0.07191614806652069, 0.14451657235622406, 0.07902853935956955, -0.8035086989402771, 0.3480266332626343, -0.2731456160545349, 0.5203015208244324, -0.9719205498695374, -0.2216523438692093, -1.0228945016860962, -0.7953587770462036, -0.012210901826620102, 0.0963960513472557, -1.6624912023544312, -0.33830755949020386, -1.5052030086517334, -0.6302485466003418, 0.5063315629959106, 0.2604590356349945, 1.5911988019943237, -0.4281545877456665, -0.7447918653488159, 0.12056703120470047, -0.908583402633667, -0.8663665652275085, 1.0376704931259155, -0.5306888222694397, -0.8383570909500122, -1.1646313667297363, -0.7322384119033813, 1.88508141040802, -0.8753751516342163, 0.6569469571113586, -0.7689837217330933, 0.4267171025276184, -0.3406491279602051, -0.308552086353302, -0.2042967826128006, 1.0649347305297852, 0.8866451382637024, 1.755162239074707, 2.184703826904297, 0.18600322306156158, -0.8479888439178467, 0.8648666143417358, 0.8091717958450317, -0.05273500084877014, 0.40818309783935547, -1.3796727657318115, 0.761661946773529, -1.650233268737793, -0.46030518412590027, 0.677393913269043, 0.4707294702529907, 0.05427871271967888, 0.4610779583454132, 1.041116714477539, 1.7876557111740112, 0.6388304829597473, 0.5706110596656799, -0.3512902855873108, -1.5187627077102661, 1.9274529218673706, -2.108412742614746, -2.0089704990386963, 0.16007426381111145, -0.29154810309410095, -0.0651918426156044, 1.664660096168518, 0.40142402052879333, 0.7035833597183228, 0.3256172835826874, -0.585895836353302, 0.6446494460105896, 0.2379683256149292, -0.16336317360401154, -1.2250839471817017, 0.47947701811790466, 1.2944865226745605, 0.6210156083106995, 0.9673284292221069, 0.04681463912129402, 0.19666679203510284, -0.8548182249069214, 0.6500494480133057, 1.0501890182495117, -0.023437460884451866, -0.8370286822319031, 1.37078058719635, -0.18699340522289276, -1.7277302742004395, -0.06581582874059677, -2.2833964824676514, 1.5542569160461426, 0.6206912398338318, -2.4166946411132812, 0.7480949759483337, 1.7414253950119019, -1.3185828924179077, -1.6638675928115845, 0.683057427406311, -0.9941128492355347, 1.4887946844100952, -0.1272701770067215, 2.1716196537017822, -1.0793129205703735, 0.31874409317970276, 0.5914469361305237, 0.39033201336860657, 1.1185137033462524, -0.6856910586357117, -1.6142911911010742, -1.345799207687378, 0.0886523649096489, -0.6092706918716431, 0.9496245980262756, 1.142569899559021, 0.29185810685157776, -0.6394559144973755, 1.5507529973983765, -0.04962289333343506, 0.8939142227172852, -0.03190891072154045, -1.0865659713745117, -1.7080049514770508, 0.17927494645118713, -0.16634000837802887, 0.1257004588842392, 1.3680393695831299, 1.5430290699005127, 0.43745332956314087, -0.9087315201759338, -0.6253435015678406, -0.1648089736700058, 0.21719269454479218, -1.9179326295852661, 0.9735254049301147, 0.2813590466976166, 0.49179625511169434, 0.5296995639801025, 1.3310400247573853, -0.1023203507065773, -0.6746892333030701, -0.5490586161613464, -0.35087257623672485, -0.02934621088206768, -1.674493432044983, 0.721693754196167, -0.27274811267852783, 1.3348217010498047, -0.04272213578224182, 0.12692026793956757, 1.3068288564682007, -0.4396996796131134, 2.326528310775757, -1.225091814994812, -0.47633567452430725, -0.9873539805412292, -0.08089606463909149, 0.8011252284049988, 0.20980329811573029, -1.3072820901870728, -1.0703753232955933, -0.6670674085617065, 1.1233093738555908, 1.1932356357574463, 0.4064386487007141, 1.0902042388916016, -0.4095819890499115, 0.3446830213069916, -0.24420584738254547, -0.13155822455883026, -0.5812044739723206, 0.12829367816448212, -1.1715387105941772, 0.10745927691459656, -1.073239803314209, 2.072040557861328, 0.13693049550056458, -0.8600752949714661, -0.17233064770698547, -0.7027292847633362, -1.0764427185058594, 0.03547355905175209, -1.9508394002914429, -0.4889831244945526, 0.7073018550872803, -1.1986862421035767, 0.08231738954782486, 0.8272786736488342, 1.8975281715393066, -0.911975085735321, 1.2557919025421143, 0.44755125045776367, 0.6067295074462891, 1.521217703819275, 0.7823647260665894, -1.668975830078125, 0.8246005177497864, -1.644264817237854, -0.6987336277961731, 1.5843532085418701, 1.2763099670410156, 0.2134220153093338, 0.4566538333892822, -0.8887459635734558, -0.8471164107322693, -1.2530088424682617, 1.1119184494018555, 0.35408902168273926, -1.9291073083877563, 0.27642518281936646, 0.6184031367301941, 1.1621044874191284, 1.2094053030014038, 0.09546414762735367, 1.1949872970581055, 0.061645347625017166, 1.641074538230896, 1.6355966329574585, 0.5063081979751587, -0.24472175538539886, 1.0789875984191895, 0.40050676465034485, -0.9880408048629761, -0.09988262504339218, 0.49312737584114075, -0.9562920928001404, -0.2962089776992798, -0.5014650821685791, -0.11949007958173752, 0.3409234285354614, 0.23259283602237701, -0.9917343854904175, 0.7410587668418884, -0.2925627827644348, -0.994990885257721, -0.34134694933891296, 1.2625668048858643, 0.28014177083969116, -1.651171088218689, -0.429965615272522, -0.771155059337616, 0.5945292711257935, 0.5501831769943237, 0.41923826932907104, -0.15417072176933289, -0.49571049213409424, 0.4597841203212738, 0.08652115613222122, -0.07721877843141556, 0.040297795087099075, -0.48259589076042175, 1.8086367845535278, 1.4613615274429321, -0.7510035037994385, 0.47704166173934937, -0.7183278799057007, -0.9501006007194519, -0.3757248520851135, 0.27059176564216614, 0.000244561699219048, 0.5974875092506409, -0.21849969029426575, 0.387956827878952, -0.04423978179693222, 0.7274889945983887, 0.21252740919589996, -0.025803938508033752, -1.3908584117889404, -2.1508913040161133, 0.6698421239852905, 1.5501121282577515, 0.47958844900131226, -0.7782750129699707, 0.2207605391740799, -0.14379727840423584, -1.2508633136749268, 0.11961425840854645, 0.5554921627044678, 0.6513733267784119, 0.24590465426445007, -0.4165687561035156, 1.0686267614364624, 2.073566198348999, -1.5852489471435547, 1.209482192993164, 0.684112548828125, -0.7775704860687256, 1.0858925580978394, -0.6955595016479492, 0.8735358119010925, -0.36007943749427795, 0.5284576416015625, -0.14076410233974457, 0.3786114752292633, 1.0396523475646973, -1.2345980405807495, -0.8839300870895386, -1.9016796350479126, -0.30910131335258484, -0.569031834602356, 0.9032198190689087, 1.5323771238327026, 1.0123646259307861, 0.5458292365074158, 0.17351582646369934, -0.5612749457359314, 0.8349524736404419, 3.062072515487671, 0.5633975267410278, -2.0404531955718994, 0.667746365070343, 2.2983009815216064, 0.5513131022453308, -0.453355997800827, 1.3197251558303833, 0.3329057991504669, -1.1158908605575562, 1.2332662343978882, -0.46139511466026306, 1.1505745649337769, -1.3751049041748047, -0.9810745716094971, 0.18657638132572174, 0.4176733195781708, -0.8739463090896606, 0.1572958528995514, 1.9414877891540527, 1.322823166847229, -0.527171790599823, -1.6489999294281006, -0.11991631239652634, 0.8607532382011414, 1.3878329992294312, -1.2757246494293213, -0.8475511074066162, 0.8345955014228821, 0.31746453046798706, 0.8257731199264526, 0.010734680108726025, -1.5847594738006592, 0.19477415084838867, -0.7191639542579651, -0.2864711284637451, 0.22165927290916443, -0.12796425819396973, 0.5001720190048218, 0.5049656629562378, -0.6848065853118896, -1.4995197057724, -1.4921338558197021, 0.09165079146623611, -0.4066421687602997, -0.4572688043117523, -0.23496142029762268, 1.1064969301223755, 0.20860032737255096, 0.33362439274787903, 1.241457223892212, -1.1325527429580688, 0.5298882722854614, 0.8970987200737, 1.9834096431732178, -1.9465993642807007, -2.101250410079956, 1.1889348030090332, -0.6738904118537903, -2.2786648273468018, -1.1732916831970215, 1.3605766296386719, -0.04757791757583618, 0.9233040809631348, 1.5302618741989136, -0.8382682800292969, 0.5420854687690735, -0.0022248588502407074, -0.34665101766586304, -1.9852588176727295, 0.026728078722953796, -0.3185601234436035, 0.6134951114654541, 0.3985944390296936, -1.3979791402816772, -0.14799991250038147, -0.07065650820732117, -0.5235608816146851, -0.08086058497428894, 0.5363799929618835, -0.9645962715148926, -0.06666549295186996, 0.5875958204269409, -1.7920005321502686, -1.0265085697174072, -0.44020137190818787, -2.5837953090667725, 0.8021255731582642, 1.0723376274108887, 0.46614813804626465, -0.06737709790468216, 0.6613175868988037, 0.8523368835449219, -0.7894220948219299, -0.6353368163108826, -0.5915597081184387, -0.9201215505599976, 0.7308627367019653, 0.34148043394088745, -1.2916085720062256, 0.2801473140716553, -1.0766700506210327, -0.2520797550678253, -0.11923469603061676, 1.3193187713623047, -0.58298259973526, -0.07614833116531372, 0.5270817875862122, 0.09233824908733368, 0.4718921184539795, -0.8757721781730652, -0.281292587518692, -0.09931674599647522, 1.379867672920227, -1.0554718971252441, -0.4039694666862488, 0.4043238162994385, 0.6083386540412903, 0.07211411744356155, 0.7814547419548035, -1.3258005380630493, 0.7215635180473328, -0.690574586391449, -0.41752493381500244, 0.4053453505039215, -0.23225975036621094, -1.0007840394973755, 0.16535820066928864, 0.14810189604759216, -0.25805842876434326, 0.9920850992202759, 0.14081090688705444, 0.7104759216308594, 2.0061137676239014, 0.4295291304588318, 0.1527767777442932, 0.9538550972938538, -0.32174214720726013, 1.3400514125823975, -1.0650920867919922, 0.9373272061347961, 0.015854069963097572, -0.9755929708480835, -0.7471216917037964, -0.6606170535087585, -1.1694862842559814, -1.537617564201355, 1.1135118007659912, -0.5912637114524841, -0.20465102791786194, -2.377140522003174, 0.9742645025253296, 0.32553720474243164, -0.7281185388565063, 1.6440355777740479, -0.3236388564109802, 0.8731026649475098, -0.5584964156150818, -0.09759417176246643, -0.6620748043060303, -0.20802441239356995, 1.6497317552566528, 0.44300606846809387, -0.2527411878108978, 0.018138468265533447, -0.8917027115821838, 0.1302778422832489, 1.4629877805709839, 0.724660336971283, 0.15874850749969482, 0.49983081221580505, -1.0484610795974731, -0.884030282497406, 0.955280065536499, -0.4956088662147522, 0.06352092325687408, -0.08588667213916779, 1.5043556690216064, 0.12107586860656738, -1.4559038877487183, 0.10682620853185654, -0.8164463043212891, -0.41829973459243774, 1.0692088603973389, -0.017168253660202026, 0.9277552962303162, 1.196424126625061, -0.636456310749054, 1.6727442741394043, -0.5797932744026184, 0.26515933871269226, -0.949451744556427, -0.2583700716495514, -0.3839118778705597, 1.1315604448318481, 0.17065772414207458, 1.385844349861145, -0.34247416257858276, -0.9870036840438843, -0.5764203071594238, -0.2839277684688568, 0.10111908614635468, -0.31108906865119934, 0.8193880915641785, -0.6244574785232544, 0.08129366487264633, -1.006083607673645, -2.293894052505493, -0.9863877296447754, 0.37015023827552795, 1.4537838697433472, 0.40313687920570374, -0.6042512655258179, -0.6241124868392944, 0.4672711193561554, 0.8353845477104187, -0.030557654798030853, -1.079750657081604, 1.2792534828186035, -0.21986739337444305, 0.9743881821632385, -1.0590580701828003, -0.5040515065193176, 0.4928842782974243, -2.017256736755371, -0.4458116292953491, -0.6046921610832214, 0.6678824424743652, -2.1500208377838135, 0.11675458401441574, -0.07631633430719376, 1.2086814641952515, -1.1019388437271118, 1.3109468221664429, 0.7604253888130188, 0.5786125063896179, -0.5458617806434631, 1.0586788654327393, -0.28396129608154297, 0.08402322977781296, -0.4001438021659851, -0.3198416233062744, -0.125849649310112, -0.8412483334541321, -0.9572256803512573, -1.6744624376296997, -0.09242305904626846, 2.264820098876953, 0.5060881972312927, -1.0663219690322876, 0.11695106327533722, 0.718226432800293, -1.6988856792449951, -0.5236324071884155, -0.48126447200775146, -0.3003194034099579, -0.6322160959243774, 0.899133026599884, 0.6273406744003296, -1.1453007459640503, 1.5045510530471802, 0.8180884718894958, -0.45820364356040955, 0.42426928877830505, -1.1409903764724731, 0.05969814956188202, 0.10065127164125443, -0.7473812699317932, -1.4344724416732788, 0.14103244245052338, -0.22429729998111725, 0.500024139881134, 0.7955123782157898, 1.2438231706619263, 0.910487174987793, 0.7609742879867554, 1.2579262256622314, 1.5099996328353882, -0.1286841481924057, 0.18011674284934998, -0.19051599502563477, -0.0979895070195198, 1.0346342325210571, -0.7942602634429932, -0.14504773914813995, -0.32946789264678955, 0.49093642830848694, -0.42995014786720276, -0.26253020763397217, -0.5447506904602051, 0.604293942451477, -0.8689536452293396, -0.4362567663192749, -1.7755175828933716, -1.553916335105896, -0.0587027482688427, 0.4592103660106659, 0.7178860306739807, -0.45335447788238525, -0.008890406228601933, 0.8144732117652893, 1.1478161811828613, -0.7885969877243042, 0.2504357099533081, 0.04574258252978325, 0.317876398563385, 1.3189048767089844, 0.5898664593696594, -0.2795313894748688, 1.1435902118682861, 0.0821392759680748, 3.0695912837982178, -2.1850621700286865, -0.023033248260617256, -0.10032444447278976, 0.31390997767448425, 1.6982173919677734, 1.379899263381958, -0.65227872133255, -0.6034388542175293, 1.5659955739974976, 1.419472575187683, -1.6684401035308838, 0.22530153393745422, -0.8368983268737793, -0.008334380574524403, 0.03336535766720772, -1.012212872505188, 2.5420703887939453, 0.6707254648208618, 1.0616581439971924, 0.10309107601642609, -0.4889170527458191, -0.3166552484035492, -1.1686266660690308, -1.5241750478744507, 1.594743251800537, -0.11786535382270813, 0.3370085656642914, 1.2469918727874756, -5.498153041116893e-05, -0.3455100953578949, 0.6793354153633118, 0.5658928155899048, -0.08268421143293381, 1.2796616554260254, 0.9275352954864502, 1.5880684852600098, -1.407485842704773, -0.25727394223213196, 0.18074442446231842, 2.1554017066955566, -0.8705191016197205, -0.36348700523376465, -1.5237611532211304, 2.1108717918395996, 1.2754652500152588, 1.9223676919937134, -0.3061385452747345, -0.7702965140342712, -0.8724339604377747, 1.8309170007705688, -1.8973493576049805, 0.08411160856485367, -0.8597830533981323, 0.4531475603580475, -0.7674424052238464, -1.4658862352371216, 0.2812100648880005, -0.4319504201412201, 0.4800652265548706, -0.48092424869537354, -0.574142575263977, 1.9163116216659546, -0.9982150793075562, 0.5462853312492371, 0.440310537815094, 0.1037631556391716, -2.0017263889312744, 1.8324971199035645, -2.167665958404541, 0.7021312117576599, -0.7802761793136597, 1.086199164390564, 0.6005563139915466, -0.8264764547348022, -0.8884096145629883, 0.2054160237312317, -0.7321856021881104, 0.007556453347206116, -0.2587200105190277, 0.5529941916465759, -0.8389750123023987, -0.028951825574040413, -0.5323510766029358, -2.04238224029541, 0.670582115650177, 1.0667603015899658, 0.8137267827987671, 1.0774993896484375, 0.947002649307251, 0.07219746708869934, -0.5905473828315735, -0.022599712014198303, 0.6158534288406372, 1.4129090309143066, 0.555272102355957, -0.8291711807250977, 1.5335054397583008, -0.991032063961029, -1.6947256326675415, -0.924882173538208, -0.07662000507116318, 1.0016958713531494, -0.36695051193237305, 1.620612621307373, -1.5776985883712769, -0.8101678490638733, -0.9534199833869934, 0.3399111032485962, 0.30827224254608154, -1.236835241317749, -0.36722689867019653, -0.5538227558135986, 0.5283523797988892, -1.7598953247070312, 1.164293646812439, -1.8825256824493408, 2.019345998764038, 0.028643034398555756, 0.955080509185791, 0.8980070352554321, -1.8066892623901367, -0.9441823363304138, -0.313308447599411, 1.0662975311279297, -1.3716589212417603, -1.4495985507965088, 1.8157516717910767, -1.1448314189910889, 1.5218040943145752, 0.8104662299156189, -0.6218048334121704, 0.617357075214386, 0.9725749492645264, 0.11514254659414291, -2.091362237930298, -0.9711249470710754, -0.6606420278549194, 2.1211578845977783, 0.0239169392734766, 2.0138561725616455, 0.2939684987068176, 1.3865991830825806, -1.2777338027954102, -0.297272652387619, 0.4238011837005615, -0.48844125866889954, -0.8307628035545349], [-2.409139633178711, 0.018337761983275414, 1.9575964212417603, 0.7947381734848022, 0.5846042633056641, 0.1821168214082718, 0.6128612756729126, -2.463859796524048, 1.865918517112732, -1.6809635162353516, -0.2750454843044281, -1.7371361255645752, 0.8568849563598633, -2.888066530227661, -2.3088388442993164, 0.6549386382102966, 0.08371499925851822, -1.6663637161254883, 1.8460960388183594, -0.47793006896972656, -0.24142788350582123, -0.9650816917419434, 1.1174333095550537, -0.0028458258602768183, -0.2156434953212738, 1.3454153537750244, -1.3327213525772095, 0.6493133306503296, -1.1787530183792114, 0.7271923422813416, 0.043353673070669174, 0.680732250213623, -2.066420555114746, -0.28541624546051025, -1.686652660369873, 1.1962941884994507, 0.06254874169826508, -0.3444547951221466, 0.5686269998550415, 0.27529624104499817, 0.8793590664863586, -0.020588263869285583, 0.8680867552757263, 0.7212414145469666, 1.6271164417266846, -0.42063790559768677, -0.5579650402069092, 0.7706778645515442, -1.7449947595596313, 1.527829647064209, -0.20180058479309082, -0.5224811434745789, -0.40136799216270447, 1.0571420192718506, 1.6418696641921997, -0.8478968739509583, 0.8209848999977112, -1.4314566850662231, -1.462996006011963, -0.8268596529960632, -0.6776880621910095, -0.36692938208580017, 0.4420579671859741, -0.4789801836013794, 0.9839705228805542, -0.5254669785499573, -0.1920185685157776, 0.832692563533783, -1.9585086107254028, 0.7128159403800964, 0.2978033423423767, 0.3930884003639221, 1.0914629697799683, 0.6095480918884277, -0.9762609004974365, -0.2455430030822754, -0.8020815253257751, 0.6690974831581116, 0.10410827398300171, 1.4830310344696045, -0.4152098298072815, 1.7255103588104248, 0.186695858836174, 0.683232307434082, 0.974748432636261, -0.3544677197933197, 1.8288919925689697, 0.5383946299552917, 0.33922991156578064, 1.488394856452942, -1.2845561504364014, -1.8608189821243286, 0.9504017233848572, -1.9130463600158691, 1.070554494857788, -1.4113836288452148, -0.8728828430175781, 0.34233391284942627, 1.4779969453811646, -1.3517149686813354, 0.6697332859039307, 1.0475876331329346, -1.5386366844177246, 1.3172507286071777, 1.85423743724823, -0.7548630237579346, 1.5242650508880615, -1.1023635864257812, 0.8340237140655518, 0.39086493849754333, 0.28899163007736206, -0.1682913601398468, -0.6520133018493652, 0.5591362118721008, -1.2524471282958984, 0.9495997428894043, 0.36720749735832214, -0.6089337468147278, -1.869949460029602, -0.6600283980369568, -1.626680850982666, 0.13044637441635132, -0.5103548169136047, 0.3009520471096039, 0.22311650216579437, -0.28865960240364075, -1.6663918495178223, -0.418749064207077, -0.07161020487546921, -0.927143931388855, 0.1848299652338028, 0.7947483658790588, -0.15685401856899261, -1.3190348148345947, 0.6475962400436401, -0.7580018639564514, -0.16133834421634674, -0.12627814710140228, -1.6903444528579712, 0.5583504438400269, 0.7667535543441772, -0.300513356924057, 0.3526887893676758, -0.11275456100702286, -0.028965823352336884, 1.4538642168045044, -0.7819182872772217, -0.24520252645015717, -1.590064525604248, 0.6604244112968445, 0.07930450141429901, 0.4924963414669037, 0.06579924374818802, -0.647601306438446, 0.41654300689697266, 1.3889875411987305, 0.3227711617946625, 1.5004138946533203, 1.3216607570648193, 0.1559724509716034, -0.9727843999862671, -0.5925719738006592, 0.09580273926258087, 0.5726475715637207, -1.04196035861969, 0.3178565800189972, -1.6711589097976685, -0.23029324412345886, -0.2943660318851471, 0.7007888555526733, 0.030874641612172127, 0.11623772978782654, 0.9148966670036316, 0.5312535166740417, -0.040538039058446884, -0.9684522747993469, -0.27861151099205017, -0.1130567342042923, -0.23980854451656342, -1.9982366561889648, -0.29803264141082764, -0.685313880443573, -0.3352207541465759, -1.0269826650619507, -0.15009872615337372, -0.4924301505088806, -0.032727815210819244, 1.785901427268982, 0.1624232828617096, -1.0364986658096313, -0.633547842502594, 0.21566995978355408, 1.1618237495422363, 0.4135296046733856, 0.10824702680110931, 0.39165639877319336, -0.2608945071697235, -0.16999447345733643, 1.1723037958145142, -0.28181955218315125, -0.002716370625421405, -0.6154850721359253, -0.6322543025016785, 1.1885507106781006, -2.610731601715088, 0.3028145134449005, -0.39131277799606323, 0.7494115233421326, -0.5820682644844055, 0.14682310819625854, -0.14206045866012573, 0.019723791629076004, -0.37251871824264526, -0.8648241758346558, 0.40154093503952026, 0.7792573571205139, -0.32110556960105896, 1.214690923690796, 0.03217332437634468, -1.0250259637832642, 1.2725743055343628, 1.456748604774475, -1.1338061094284058, -0.11609125137329102, 1.4939208030700684, 1.2279233932495117, 0.7325694561004639, 0.43043220043182373, 0.06566192209720612, -1.1163829565048218, 0.42912420630455017, 2.0948991775512695, 0.6081047058105469, 0.1157703697681427, 0.23949716985225677, 0.12256038188934326, 1.7644466161727905, 1.4917610883712769, -2.7257463932037354, -0.729303240776062, -1.4512931108474731, 1.5720709562301636, 0.37748974561691284, 0.12449351698160172, 0.3858011066913605, -0.12374135851860046, 0.5749391913414001, -0.9840396046638489, -0.5102618336677551, -0.847682774066925, -0.9963576197624207, 1.0267658233642578, 0.01267951913177967, -0.534740686416626, 1.538176417350769, 0.23768432438373566, 0.35028669238090515, -1.0617527961730957, 0.8704830408096313, 0.6067361831665039, 0.9298229217529297, 0.5455141663551331, 0.4829787015914917, -0.7764022946357727, -0.4939063489437103, 1.8897700309753418, -0.26613670587539673, 0.6484711170196533, 1.3493422269821167, -0.17539705336093903, 0.2143574059009552, -1.0496227741241455, 0.341205894947052, 0.5046743750572205, -1.6454614400863647, -1.0883922576904297, 0.10949544608592987, -0.46943435072898865, 1.5141574144363403, 0.2334222048521042, -1.2683371305465698, -0.16671065986156464, -0.7992560267448425, -0.6121935248374939, 0.5375050902366638, 0.2328759878873825, -2.450164794921875, 0.6822432279586792, 0.34960857033729553, -0.6232443451881409, 0.19375084340572357, -0.048621632158756256, 1.5648897886276245, 0.25400054454803467, 0.40090227127075195, -0.3002631366252899, 0.13347406685352325, 1.3006749153137207, -1.7837252616882324, 1.7375514507293701, -1.8365654945373535, 0.4395122826099396, -1.0518807172775269, 0.22401568293571472, -0.6729691624641418, -1.6414464712142944, -0.08394655585289001, 0.2041819840669632, -1.0982542037963867, -0.09345390647649765, -1.7710553407669067, -0.37366095185279846, 0.9969271421432495, 0.2089969515800476, 1.407631754875183, 1.8419841527938843, -2.800455093383789, -0.0015151480911299586, -2.0010182857513428, 0.8761919140815735, 0.8924758434295654, -1.4773937463760376, 0.4174738824367523, -1.0405504703521729, 0.17540065944194794, -0.14915986359119415, -0.1143648624420166, 1.838984727859497, -0.24907627701759338, -0.4454157054424286, 0.5753186941146851, 1.06462824344635, -0.6784834861755371, -0.7899128794670105, 0.8553816080093384, -0.5677958130836487, -1.022269368171692, 1.4047703742980957, -0.6778441667556763, -1.6858208179473877, 1.308303713798523, -0.5713675618171692, -1.969251036643982, 1.9820234775543213, 1.3220070600509644, -1.228153944015503, -0.7294698357582092, 0.23687738180160522, -0.07334388047456741, 0.4128492474555969, 0.33644646406173706, 1.638971209526062, 0.5649966597557068, -0.9941797256469727, 0.20207896828651428, -1.4051637649536133, 0.8035224676132202, 0.4527466595172882, 0.7930107712745667, 0.5629389882087708, -0.5306183099746704, -0.021864352747797966, 1.5387083292007446, 0.7721081972122192, -0.6066175103187561, -0.7398415207862854, -0.32334715127944946, 0.9583232998847961, -0.6924800276756287, 1.1810845136642456, -0.9936739802360535, 0.9304696917533875, 1.026842713356018, -0.4136350452899933, -1.0674248933792114, -0.3655151426792145, -0.18310841917991638, 0.346934050321579, 0.5385239720344543, 0.9548683762550354, 0.7499867081642151, -0.8354450464248657, 1.4493237733840942, 0.2664318382740021, -1.436234951019287, -3.0920560359954834, -0.16646340489387512, 1.2158299684524536, -0.4253385663032532, -0.2808447480201721, -0.1964379847049713, -0.5809097290039062, 0.02109004184603691, -0.3656589686870575, -0.31037893891334534, 1.0183523893356323, 0.3180907964706421, -0.49472373723983765, -0.27895164489746094, -1.003515601158142, 0.4176500737667084, -0.5334067940711975, -0.8832887411117554, -0.886189877986908, -0.20553041994571686, 1.2583194971084595, 0.23439113795757294, 0.9612656831741333, -0.6086328029632568, -0.44033747911453247, -0.9166939854621887, -1.0339851379394531, 1.1355973482131958, -0.1648208349943161, -0.4532226324081421, -0.07170575112104416, 1.4579674005508423, -0.4778098165988922, 0.6323534846305847, -0.783915638923645, -0.6763696074485779, 1.0594508647918701, 0.3066055178642273, -0.4903147220611572, -1.0077924728393555, 0.8221880197525024, 0.8536312580108643, -0.29372352361679077, 1.0146232843399048, 1.0393613576889038, 0.25560471415519714, 1.2810711860656738, 0.5028572678565979, -1.4808686971664429, 0.6250202059745789, 0.2891542911529541, 0.1999996304512024, -0.25123316049575806, -0.0017021071398630738, 0.5545493960380554, 0.42284131050109863, -1.7288389205932617, 0.8254740238189697, 0.5665861368179321, 0.4047507047653198, 1.3272244930267334, -1.2751563787460327, 0.059773217886686325, -1.1727545261383057, -0.8168622851371765, -0.09066608548164368, 0.39015960693359375, 0.9041581749916077, -0.16168804466724396, 1.0730993747711182, -0.5204178094863892, 0.9185234904289246, -1.2648731470108032, -1.1654212474822998, 0.2690201699733734, 0.12071014195680618, -0.7480150461196899, 0.010822252370417118, 1.473460078239441, 1.8610703945159912, 0.6106621623039246, 1.0662319660186768, 0.6977349519729614, 1.3852014541625977, -0.5132178664207458, -0.56331467628479, -1.9821642637252808, -0.6419015526771545, 2.1521501541137695, 1.0668421983718872, -0.24503843486309052, 0.515478789806366, -1.4933242797851562, -0.07438994199037552, 1.368263602256775, -2.1188414096832275, 0.43404555320739746, -2.5219287872314453, 0.9033920168876648, 2.147385835647583, 1.2460005283355713, 1.2050433158874512, 0.8981226682662964, -0.03484223410487175, 3.0361504554748535, 0.65617436170578, 2.106706142425537, -0.28741613030433655, -0.044026345014572144, -2.2869508266448975, -0.6797110438346863, 0.6229016780853271, 0.8186308741569519, -1.3868013620376587, 1.199472188949585, -0.14759045839309692, -0.36852917075157166, -1.1843137741088867, -0.666759729385376, -0.1600540280342102, 1.2147784233093262, -0.7158329486846924, -0.1937660425901413, 1.1617881059646606, 0.5328253507614136, -0.8959888219833374, -0.5823166370391846, -0.14289194345474243, 1.4587866067886353, -0.6349932551383972, 0.5725285410881042, 0.1800849437713623, 2.504554033279419, 0.8366725444793701, -0.12033305317163467, 0.033077362924814224, 0.650044322013855, -0.9857316613197327, 1.1002936363220215, 0.22593389451503754, 0.1542944461107254, 0.9051398634910583, -0.9556867480278015, 1.411872386932373, -1.0965330600738525, 1.458532452583313, 2.442485809326172, 0.3427467346191406, -0.6839553713798523, -0.9457799792289734, 0.38839244842529297, 0.7163076996803284, -0.03241737559437752, 0.6708581447601318, -1.2208750247955322, -0.3232981264591217, 0.19212350249290466, -0.5127593278884888, -0.2671334147453308, -0.9997133612632751, 1.7027783393859863, -0.04891986399888992, -0.594556987285614, 1.2332730293273926, -0.017226124182343483, -0.1737261414527893, -0.20446540415287018, -0.24333202838897705, 0.6103510856628418, 0.6669343709945679, -0.2907305359840393, 0.39667558670043945, 0.7808119058609009, -0.8600068688392639, -0.3317653238773346, -0.03714234381914139, 1.1950706243515015, 0.31172895431518555, 2.859848976135254, -1.2549090385437012, -1.0510426759719849, -0.6172307133674622, 0.3449077308177948, -0.5534095168113708, 0.6965263485908508, -1.1801254749298096, -1.1540149450302124, -0.9649999141693115, -1.4907493591308594, -0.9121580719947815, -0.1128976047039032, 0.1196851059794426, 0.00018002443539444357, 1.0585591793060303, 0.14094482362270355, 1.7680909633636475, 0.335004597902298, 0.48249906301498413, 1.1052907705307007, 0.6412487030029297, -0.509154736995697, -0.1717551052570343, -1.1921656131744385, 0.7403304576873779, 0.477029025554657, -1.364532709121704, -1.732115626335144, -0.8103564381599426, -1.6556326150894165, -0.8543586134910583, -1.3678723573684692, 0.2201671600341797, -0.15448561310768127, 0.023351484909653664, -0.41916534304618835, -2.2346222400665283, 0.7594356536865234, -0.5188614726066589, -0.8907099962234497, 0.09410853683948517, -0.49092093110084534, 1.0130609273910522, 0.18030302226543427, 1.292712926864624, 0.4939057230949402, -0.5356317162513733, -0.7515638470649719, 0.025118328630924225, -0.8384608626365662, 0.24499960243701935, -0.34032872319221497, 2.1727020740509033, 0.739974319934845, -0.08485034108161926, 2.1494133472442627, 0.6942957043647766, 1.1749354600906372, 1.1843278408050537, 0.01160801388323307, 0.5853203535079956, -0.31334245204925537, -2.000798463821411, -0.4335534870624542, -1.002423882484436, 1.5673853158950806, 0.036509864032268524, -1.9206186532974243, 0.051241807639598846, -1.2398370504379272, -0.32247766852378845, 1.6788341999053955, 0.5545796155929565, 0.019533446058630943, -1.0520952939987183, 1.913425326347351, -0.11817681044340134, 1.2803964614868164, 0.3193211853504181, 1.179170846939087, 0.25189512968063354, -0.031136780977249146, 1.2968382835388184, 0.26933401823043823, -1.160654902458191, -0.9425106644630432, -0.2955482602119446, -0.29938530921936035, 0.420814573764801, -0.5376899838447571, 1.1752170324325562, -1.195487141609192, 0.43081536889076233, -0.7788257598876953, -0.3690028488636017, 0.8167558908462524, -0.7582749724388123, 2.0123026371002197, -1.8014754056930542, -1.948850154876709, -0.05178287625312805, 0.7515475153923035, -0.028258316218852997, -0.9481195211410522, -0.6452633738517761, 1.314345121383667, -0.27181071043014526, -0.5798779129981995, -0.04278748109936714, -0.8850648999214172, 0.31615546345710754, -0.28287309408187866, 0.4667704403400421, -0.327217698097229, -0.8076876401901245, -1.7806915044784546, -0.6811426281929016, -0.294290155172348, 0.29359695315361023, -0.694808304309845, 0.017430350184440613, -1.567068099975586, -2.430424213409424, 1.9810895919799805, 1.1645525693893433, 0.9538675546646118, -1.185814380645752, -0.7470734119415283, 0.7717916965484619, 1.655521035194397, 0.7149505615234375, 0.9116851091384888, -0.9967094659805298, 1.639692783355713, 0.7372961640357971, 1.1329209804534912, 1.438323736190796, -0.17999637126922607, 2.575413703918457, 1.052200198173523, -0.02790159545838833, -1.2554327249526978, -0.5462140440940857, -2.1125869750976562, -1.4337964057922363, 0.4886000156402588, 0.3659136891365051, -0.06574415415525436, -0.34589695930480957, 1.103756070137024, -2.47694730758667, 1.1083911657333374, 1.004372239112854, 0.15050676465034485, -0.9309718012809753, 1.028031826019287, -1.3572585582733154, -0.4287610352039337, -0.5945073962211609, -1.3991798162460327, 1.4545655250549316, 1.1717075109481812, 1.0757758617401123, -1.7835454940795898, -0.46451810002326965, -0.7467357516288757, -0.4621546268463135, -1.883478045463562, -0.15033428370952606, -1.272331714630127, 0.8594704866409302, 0.3470311462879181, 1.203495740890503, -0.2634318768978119, 0.4977812170982361, -0.8730829954147339, -1.071561574935913, -0.24326376616954803, 0.1359204798936844, -2.051248550415039, 2.574777603149414, -2.406740665435791, -0.8277033567428589, 0.3285507559776306, -0.06789220869541168, 0.31030163168907166, 1.0247026681900024, 1.3643656969070435, -0.361995130777359, 1.1679857969284058, 1.2334132194519043, -1.6576735973358154, 0.27767953276634216, 0.6633574962615967, 0.2513297200202942, -0.17294342815876007, -0.6561406850814819, 2.013125419616699, 1.2087408304214478, -0.528997540473938, -0.690787136554718, 0.6556360721588135, -1.1006498336791992, 0.43429213762283325, 1.1129169464111328, 1.680008888244629, -0.4036096930503845, 1.3985848426818848, -1.4311805963516235, 0.6479015350341797, -1.515994668006897, -1.3048783540725708, -1.5744808912277222, 0.3007908761501312, 0.31926220655441284, -0.6451101899147034, -0.26323866844177246, -0.44204452633857727, -0.5220965147018433, -0.3063512444496155, -1.1346296072006226, -0.620542585849762, 0.0750378742814064, 0.0584862045943737, -0.4137825667858124, -0.7192936539649963, 1.21396005153656, -1.1821119785308838, 1.2462011575698853, 0.9566526412963867, 0.45323875546455383, 0.29410257935523987, 0.49352386593818665, 0.9847635626792908, -1.278306245803833, -1.0711482763290405, 0.08391149342060089, 1.2036466598510742, -0.6100154519081116, -1.2083032131195068, 0.49841392040252686, -0.7051426768302917, 0.8819461464881897, 0.5041278004646301, 1.0181519985198975, -0.08659131824970245, 1.2907342910766602, -0.8967079520225525, 0.9652047753334045, 2.0452632904052734, -1.301761269569397, -0.7876947522163391, -0.5883815288543701, -0.3905782699584961, 1.1958472728729248, -0.5190948247909546, -0.27331048250198364, 0.6074643731117249, -0.649045467376709, -0.3667762279510498, 0.9668756723403931, 0.3530849516391754, -0.06533198058605194, -0.9639670848846436, 1.586929202079773, -0.6065889596939087, 0.11020006984472275, -0.7850374579429626, -1.9746572971343994, 0.09322696179151535, -0.4371055066585541, -0.5651805400848389, -0.7088212966918945, 0.2879694402217865, 0.26187995076179504, -0.4048202335834503, -2.487089432179346e-07, 0.33432111144065857, 0.7589027881622314, 0.2728155851364136, 0.23343591392040253, 1.6503291130065918, -0.01865144446492195, 0.08936426043510437, -0.013558461330831051, 0.8629935383796692, -0.4787364900112152, -0.8333873152732849, 0.7409690022468567, 1.399449348449707, -1.371997594833374, 1.562251091003418, -0.29554373025894165, 0.6666991710662842, 0.18107959628105164, 0.36233019828796387, -0.5090999007225037, 0.8033487796783447, 0.9256430268287659, -0.9625698924064636, -1.5034363269805908, 1.8639745712280273, 0.1485486924648285, 1.4683524370193481, -0.564070999622345, 0.5467153787612915, 0.5380425453186035, -1.5110996961593628, 0.9960806369781494, 0.3754050135612488, 0.25668421387672424, 0.19152408838272095, 0.6197723746299744, -0.04070165008306503, -0.11269393563270569, 1.1330127716064453, 0.3760221600532532, -0.2092699557542801, 2.536452293395996, 0.3152853548526764, 1.0513569116592407, -1.1168922185897827, 0.09601987153291702, 0.18762774765491486, 0.5873653292655945, 1.548237919807434, -0.5991320013999939, -0.14210641384124756, 1.634482741355896, -0.5701489448547363, 0.27107593417167664, -0.23844629526138306, -0.23943927884101868, 0.28143736720085144, -0.6601400971412659, -0.0739317238330841, -0.33764222264289856, 1.5704900026321411, -0.4802371859550476, -0.8271145820617676, 0.13739676773548126, -1.2464231252670288, -0.9885178804397583, 0.2673676013946533, -0.33759719133377075, 0.9293904304504395, -0.6395522952079773, -0.8930238485336304, -1.2811956405639648, 0.39824020862579346, 1.321217656135559, 2.794860601425171, -0.8596578240394592, -0.48479562997817993, 0.8279940485954285, 1.0944905281066895, 1.170739769935608, -0.8031560778617859, 0.5918903350830078, 0.28185638785362244, -1.0355981588363647, 0.4733676016330719, 1.950289249420166, 0.8700717687606812, -0.544959306716919, 0.059982895851135254, 0.7590116262435913, -1.4295806884765625, 1.2948994636535645, 0.4834471344947815, 0.5610874891281128, 0.8237645030021667, -0.6933517456054688, -2.202094316482544, -0.5101613998413086, -1.0774247646331787, -0.022616583853960037, 0.894799530506134, 1.9407025575637817, 0.0175175778567791, -0.15540136396884918, 0.3480960726737976, 1.3907390832901, 0.27347901463508606, 1.4897964000701904, -1.189436435699463, -3.129085063934326, -1.2244799137115479, 0.33435899019241333, 0.7558088302612305, -0.6551322340965271, 0.8935422301292419, 0.06795333325862885, 0.43113261461257935, -0.18726377189159393, 0.39306411147117615, -0.0370350107550621, -0.7082602977752686, -0.4971098005771637, -0.06600164622068405, -1.929176926612854, 1.7247885465621948, 1.1028578281402588, 0.8757049441337585, -0.32761701941490173, 0.12294376641511917, 0.95566326379776, 0.2566695809364319, -0.029416661709547043, -0.7680068016052246, 0.2692233622074127, 0.8235173225402832, -0.9585339426994324, -0.6137247681617737, 0.04778263345360756, 0.5270317196846008, -0.10254038125276566, 1.5225303173065186, 0.8771611452102661, 0.21566377580165863, -0.7569490671157837, 1.2488468885421753, 1.0820646286010742, 0.44474437832832336, 1.4015381336212158, 1.6249042749404907, 0.27102935314178467, -0.5084502696990967, 1.6132112741470337, 0.9170270562171936, -1.0112206935882568, -0.9195793867111206, 0.2765820622444153, 0.20806656777858734, 1.5062273740768433, -0.6713167428970337, 0.28005674481391907, -0.011585688218474388, -1.1955056190490723, -1.3394416570663452, 0.8148629069328308, -3.1222994327545166, 0.6717281341552734, 0.6356017589569092, 0.1368032693862915, 1.3667689561843872, -0.7874078750610352, -1.1270490884780884, 0.4595760703086853, 1.329097867012024, 1.774239420890808, -1.1217057704925537, 2.9760611057281494, -0.4012047052383423, 1.338903784751892, 1.3070244789123535, -1.6031430959701538, -0.04332902282476425, -0.12448807060718536, 0.1270325481891632], [0.17709580063819885, -0.014130009338259697, 0.2874143123626709, -0.9892566204071045, 0.296228289604187, 1.3318512439727783, -0.40676379203796387, -0.17440739274024963, -0.22125321626663208, -1.432949423789978, 0.22606834769248962, 0.7277154922485352, 0.2110954374074936, -0.1708172708749771, -1.5897613763809204, 1.2082945108413696, 0.031929437071084976, 0.3644189238548279, -0.23497623205184937, 0.3239717185497284, 0.5587775707244873, 2.1813552379608154, 1.6276389360427856, -0.42586448788642883, 0.04270728677511215, 1.7032835483551025, -0.3454464077949524, -0.5371333956718445, -0.8168081641197205, 0.6367995142936707, 0.36806532740592957, -0.06918974965810776, 0.5196710824966431, -1.0055439472198486, -0.2627471089363098, 0.9452877044677734, -0.99045330286026, -0.8487115502357483, -1.2223682403564453, 0.6355302333831787, 1.2135512828826904, 1.46602201461792, -1.9308035373687744, 0.5646710395812988, -1.4187631607055664, 0.545630693435669, 0.42228594422340393, -0.07129674404859543, 0.9215587377548218, 0.24042439460754395, 0.1295681744813919, -1.7688837051391602, 1.9864373207092285, -1.198628306388855, -0.09262028336524963, -1.1784231662750244, -0.49417588114738464, 1.7656404972076416, -1.9688189029693604, 0.02788788266479969, -0.17592494189739227, -0.5350722670555115, 0.5438074469566345, 0.8354313969612122, 1.8245781660079956, -0.11389526724815369, -1.1852457523345947, 0.1249513179063797, -0.3442116975784302, 0.7907552719116211, 0.12906940281391144, -0.6355515718460083, -0.5690212249755859, 1.3219053745269775, 1.0078402757644653, 0.7991105914115906, 0.822199285030365, 0.02710050716996193, -1.2069119215011597, -0.31176480650901794, 1.2424933910369873, -1.360106110572815, -0.08947235345840454, 1.2055617570877075, -0.6561343669891357, 0.028309140354394913, -1.251490592956543, -1.284938931465149, 1.3655760288238525, -0.39657074213027954, 0.23059052228927612, -1.2376399040222168, 2.5225625038146973, -0.6387475728988647, 0.7786297798156738, -0.5254951119422913, -3.0156350135803223, -0.6767255663871765, -0.29541298747062683, 0.14252395927906036, -1.7130615711212158, -0.5846218466758728, -1.4156432151794434, 0.8491231203079224, 0.4974229037761688, -0.13852061331272125, -1.0744993686676025, 0.5819096565246582, 0.4566766321659088, -0.15774232149124146, 1.0669883489608765, -1.283087134361267, -0.5710288882255554, -1.3024042844772339, 1.0929242372512817, -0.8550755977630615, -0.23507773876190186, -0.5608121156692505, 1.098732829093933, 2.302460193634033, 2.9852287769317627, -0.465823769569397, 1.3511114120483398, -0.5631334185600281, -1.185893177986145, -0.9422506093978882, -1.0000897645950317, -0.9294227957725525, -0.9870717525482178, -0.11118407547473907, -1.9417781829833984, 0.6184917688369751, -1.2780191898345947, -0.3144072890281677, -0.8924755454063416, -0.5827135443687439, -1.2132985591888428, -0.9191250205039978, 1.0900110006332397, -1.0748140811920166, -1.1771373748779297, 1.2290374040603638, 0.2168133407831192, -1.4778450727462769, 0.06244687736034393, -0.24518226087093353, -1.157465934753418, -0.6957162618637085, 0.4623408317565918, -1.0043238401412964, 0.9693964123725891, -0.2361917793750763, -0.008583476766943932, -0.7532094120979309, 0.1724311113357544, -0.3897334635257721, 0.4450099766254425, -0.17617057263851166, 1.4672467708587646, -0.7674540877342224, 0.04601574316620827, 2.7092504501342773, -0.5596902370452881, -0.5425568222999573, 2.044477701187134, -0.09230808913707733, 0.8651118874549866, 0.8078505992889404, 0.15160271525382996, -1.5991402864456177, 0.02718389593064785, 2.061936855316162, -1.1297733783721924, 0.4040498733520508, 0.8947756886482239, -0.27428385615348816, -0.3398491144180298, -1.6207540035247803, 1.0963091850280762, 0.27995771169662476, -1.7567822933197021, -0.54277104139328, -0.9597474336624146, -1.332098126411438, -0.4520724415779114, 1.6738157272338867, -0.15403982996940613, -0.675751268863678, -0.5825953483581543, 1.2454967498779297, -1.561084270477295, 0.9871214032173157, -0.01878368854522705, 1.9944639205932617, -0.929008424282074, 0.4989529848098755, -0.3383373022079468, -0.6735372543334961, 0.7804391384124756, 0.7737974524497986, -0.3051457405090332, 0.07573272287845612, -0.2705031931400299, -0.7263910174369812, -0.8428798317909241, 0.2250238060951233, 0.4272138774394989, 0.6503667235374451, 0.5673165321350098, -1.3252452611923218, -0.6761486530303955, -0.8784608244895935, 1.486735463142395, 0.6091300845146179, 0.861851692199707, -0.20786966383457184, -2.14463210105896, -1.182318925857544, 0.09603673964738846, -0.4943118095397949, -0.468526691198349, 0.20968836545944214, -1.0677062273025513, 1.7627848386764526, 1.3899321556091309, -0.6665806770324707, -0.3241012692451477, 1.1860798597335815, -1.6346042156219482, 0.864025890827179, 1.6404763460159302, -1.5161060094833374, -0.09300892800092697, 1.0494095087051392, -0.8756311535835266, -0.8305370211601257, 1.0542155504226685, 1.8722213506698608, -0.1889645904302597, 0.0014196919510141015, -0.851639986038208, 1.0071953535079956, -0.548019289970398, -1.623558759689331, -1.279920220375061, -0.4157714247703552, 0.808742344379425, 0.08830618858337402, -0.08149933815002441, -0.4532833397388458, 0.6358867287635803, -0.47239091992378235, -0.09055018424987793, 0.5646358132362366, 1.101531744003296, 0.2782330811023712, 0.4818557798862457, 0.6843416094779968, 1.0660187005996704, -0.5141878724098206, 0.19124610722064972, 1.8444830179214478, 0.03116009756922722, 1.0944347381591797, -0.42779839038848877, 1.581012487411499, 0.15095257759094238, 0.018414800986647606, -0.7576329708099365, -1.2592415809631348, -0.18650974333286285, -2.0858371257781982, -0.20229677855968475, 0.7199364304542542, -0.6562820672988892, -0.21408818662166595, 1.5439084768295288, 0.8393996953964233, -0.8831757307052612, -0.23065535724163055, -0.37487396597862244, -0.1355968564748764, -0.6857284307479858, 1.2338162660598755, 0.8954266309738159, -1.181741714477539, 1.056146264076233, -0.7084251046180725, 1.1082696914672852, 0.3013538718223572, -0.0707968920469284, 0.7764131426811218, 0.7742924690246582, -0.7490399479866028, -0.10070015490055084, -0.8000411987304688, -0.38657593727111816, -0.3855826258659363, 0.7617278099060059, 0.07229948043823242, -1.4168120622634888, 0.2894328832626343, -0.3737950026988983, -0.99144047498703, 1.7169458866119385, 0.07923457026481628, -0.7623751163482666, 0.8867523074150085, -0.5683652758598328, -1.4790890216827393, 2.2644996643066406, -0.5027942061424255, -1.1959328651428223, -1.2637524604797363, -0.21862922608852386, 1.6550970077514648, 0.36878418922424316, -1.9667854309082031, 0.5840777158737183, 1.019036889076233, 1.8839919567108154, 0.3922124207019806, -1.3854200839996338, 0.6867539882659912, 0.3093869388103485, 1.4682165384292603, 0.6961309909820557, -0.8161651492118835, -0.13989616930484772, -0.08408845216035843, 0.9871509075164795, -0.7036615014076233, 0.0975133404135704, 0.518072247505188, 1.548672080039978, -0.16329781711101532, -0.48542213439941406, -0.01035247277468443, -0.2951676547527313, -1.3055490255355835, -0.20775490999221802, 0.598172664642334, -0.023701105266809464, -0.8720874786376953, 0.0764634981751442, 1.047620177268982, 0.10265669971704483, -0.8390477299690247, -0.7269138097763062, -1.0322849750518799, -0.6052857041358948, 0.4484705924987793, -0.6880067586898804, -1.1908236742019653, 0.8090184926986694, -0.03676530718803406, 0.4229479432106018, 0.4621114432811737, -1.8379979133605957, -0.8652093410491943, 0.7765164971351624, -0.023497356101870537, -0.35597437620162964, -0.94626784324646, -2.5430033206939697, 0.7828489542007446, -0.15834996104240417, -0.1194673478603363, 2.0976476669311523, 0.49989232420921326, -2.1924197673797607, 0.3646439015865326, -0.8928853869438171, -0.38930463790893555, 1.3015824556350708, -1.087728500366211, 0.2937602400779724, 1.3408290147781372, 0.9239097833633423, -0.43167644739151, 0.8574591875076294, 0.5364038944244385, -0.3941788971424103, -0.7104772329330444, 1.3041118383407593, 1.3861563205718994, 0.5530629754066467, -0.5360016226768494, -0.09729339927434921, -0.7038803100585938, 1.8766616582870483, 0.3285285234451294, -1.599877119064331, -2.347653865814209, -0.846306562423706, 0.11572325229644775, 1.739592432975769, 0.1402626782655716, -0.5276487469673157, -0.46683841943740845, 0.5605375170707703, 0.034610770642757416, 0.789940357208252, -0.7261512279510498, -0.6911364197731018, 2.3064723014831543, -0.2099561244249344, 1.2482326030731201, -0.5154204368591309, 0.6424541473388672, -0.37647897005081177, -1.1523761749267578, 0.9296952486038208, 0.05665270611643791, 1.9259006977081299, -0.19163966178894043, -2.8601038455963135, 1.9704327583312988, 0.11899982392787933, 0.36925822496414185, 0.31186509132385254, -0.36010026931762695, 0.6246405839920044, -0.006306477356702089, -0.967894971370697, 0.04336157068610191, -0.22555847465991974, 0.7525165677070618, -0.7614544630050659, 2.891622543334961, 1.473685383796692, 0.11210369318723679, -0.04236457869410515, 0.5838269591331482, -0.7206411957740784, 2.656245708465576, 1.2853221893310547, 1.1351392269134521, -0.460050493478775, -0.6681678295135498, 0.21762977540493011, 0.3156808614730835, 0.9113802313804626, -1.225426435470581, 1.4788511991500854, 0.2077171951532364, -1.2654434442520142, -0.857900857925415, 0.9319620728492737, 0.6320770382881165, -1.0028396844863892, -1.0407047271728516, 1.2263113260269165, -0.5570386648178101, 0.7299211621284485, 1.2816795110702515, 1.0470348596572876, 1.2808847427368164, 0.08123146742582321, -0.33656540513038635, 0.8995718359947205, 1.3209630250930786, -0.7704679369926453, -0.7017585039138794, 2.360895872116089, -0.17562879621982574, 0.47895342111587524, 1.866873025894165, -2.1092560291290283, 1.1013752222061157, -1.6267964839935303, 0.7340202927589417, -0.45242804288864136, -0.7006005644798279, -0.7205706238746643, -1.08027184009552, 0.4966740906238556, 1.339754581451416, 1.0492545366287231, -1.3814760446548462, -1.4606181383132935, -1.340199589729309, -1.2660841941833496, -2.206247329711914, -0.18807430565357208, -0.4027395248413086, 0.4669695794582367, 0.010290516540408134, -0.2719094753265381, 0.28607213497161865, 0.37200960516929626, -1.9851044416427612, 0.5541390180587769, -1.052807092666626, -2.010951042175293, -0.21292218565940857, 1.1903883218765259, -0.31351903080940247, 0.734778881072998, 2.2400028705596924, -1.3770980834960938, 0.6376686096191406, 0.8898133635520935, -0.21444885432720184, 1.8136364221572876, -0.1474883258342743, 1.070591926574707, -0.761572539806366, 1.2641980648040771, -0.9010687470436096, -0.48922985792160034, 0.7540855407714844, 1.0350559949874878, 1.096947431564331, -1.9641083478927612, -0.029287118464708328, -1.0298105478286743, -0.5845353603363037, -0.8679128289222717, 0.7905160188674927, -1.6875288486480713, -1.7631359100341797, -0.07305492460727692, 0.10420548915863037, -0.2238682061433792, -1.4506765604019165, -1.611725926399231, 1.7047516107559204, -0.3575636148452759, 0.05391823872923851, -2.0924177169799805, -0.6259036660194397, -1.3819482326507568, 0.6077805757522583, -1.4131771326065063, -1.3927031755447388, 0.4126668870449066, 1.4667092561721802, 2.0476181507110596, 1.8307597637176514, -1.5074654817581177, -0.4581443965435028, 0.9486792087554932, -0.035422030836343765, 0.24743564426898956, -0.15160596370697021, -1.4155747890472412, 0.039562053978443146, 1.678342342376709, -0.2156948447227478, 1.0359125137329102, 0.6668165922164917, -1.1173245906829834, -0.04449733719229698, -0.8436115980148315, -0.07073082774877548, -0.1294236034154892, -0.35851484537124634, 0.14459960162639618, -1.0477930307388306, -1.023590326309204, 0.2758382260799408, -2.219172239303589, 1.2605899572372437, 0.8027397990226746, -0.7362329959869385, -0.72429358959198, -0.034039054065942764, -0.4544033408164978, -1.0789179801940918, 0.3330383896827698, 1.2798746824264526, -0.26312243938446045, 0.4511478543281555, 0.6077144145965576, -0.36982741951942444, 0.033460184931755066, 0.23330193758010864, 1.2123256921768188, 0.32912105321884155, 0.8483668565750122, 0.2618559002876282, -0.5782570242881775, -0.05637902021408081, 1.4863702058792114, 1.139454960823059, -0.4139476418495178, 0.20067283511161804, -0.34015992283821106, 0.5826536417007446, 1.0235846042633057, -1.9385204315185547, 0.04219256713986397, 0.320818156003952, 0.9398185014724731, 1.1054818630218506, 1.343766689300537, 1.4155038595199585, 1.0280267000198364, -0.9527326822280884, 0.18271403014659882, 0.330573707818985, -0.09935155510902405, 0.6891325116157532, -0.6554017663002014, -1.4719531536102295, 0.5263511538505554, 0.5626176595687866, -0.6104826331138611, -0.04727642610669136, -0.7683808207511902, 1.6026087999343872, 1.1649646759033203, 0.4024747610092163, 1.6068193912506104, 1.5977303981781006, 0.27025794982910156, -0.9608996510505676, 0.6940965056419373, 1.5239311456680298, 0.09924433380365372, -2.3448104858398438, 1.3348921537399292, -0.9873864054679871, 0.91428542137146, 0.45542940497398376, -1.4536534547805786, -0.11186076700687408, 0.6207312941551208, -0.1506473422050476, 0.26265037059783936, 0.38390523195266724, 0.05254874750971794, 0.15534436702728271, -1.1440271139144897, -0.2546774744987488, -0.41131913661956787, -1.6933242082595825, -0.22150547802448273, 1.1763075590133667, 0.32486802339553833, 1.194702386856079, 2.150357246398926, -0.9289253950119019, -0.060798630118370056, 0.3079373836517334, 0.15935181081295013, -1.9633456468582153, 1.7976405620574951, -1.7638063430786133, -1.7233426570892334, 0.13177357614040375, 0.34559136629104614, -0.1912352293729782, 0.035389307886362076, 0.28165215253829956, -0.5221439003944397, 0.5553695559501648, 1.365620732307434, 0.038425665348768234, 1.1107302904129028, -0.34489354491233826, -1.8899301290512085, 1.1268842220306396, 1.6698698997497559, 0.03522460162639618, 0.10457420349121094, 0.8876533508300781, 1.1446189880371094, -0.050765570253133774, 0.11653496325016022, -0.9260749816894531, -1.3436909914016724, 0.9571830630302429, 1.29128897190094, 0.8943896889686584, 0.6957646608352661, -0.5178104043006897, 0.405061811208725, -0.2511543929576874, -0.21781279146671295, -0.6598742008209229, -1.218050241470337, -1.095916986465454, 0.26871147751808167, 1.0906623601913452, -0.9125910401344299, -0.7257749438285828, -0.7471861839294434, -0.0745631754398346, -0.1524500995874405, -0.34140512347221375, -0.23716115951538086, 0.005098923109471798, 1.7069391012191772, 0.3302731513977051, 0.5666720867156982, -0.6782277226448059, 2.4790902137756348, -0.469442218542099, 1.3496090173721313, -0.9605998396873474, 1.6965794563293457, -1.2664570808410645, 0.5690436959266663, -0.030711935833096504, -0.704775869846344, -2.1465203762054443, -0.8987747430801392, -0.7228264808654785, -0.10336342453956604, -0.2033284306526184, -0.3808348476886749, 0.9400209784507751, -0.29309767484664917, -0.06514992564916611, 0.5691927671432495, 0.5347477793693542, 0.1965271234512329, -0.15244579315185547, -0.6624107956886292, -1.0737574100494385, -0.4383569359779358, 1.557613730430603, -0.8683778047561646, 1.05210542678833, -1.0285046100616455, 0.045709509402513504, -0.8332847356796265, -0.5038577318191528, 0.09760314971208572, 2.138201951980591, -1.3050734996795654, 0.8619048595428467, 0.552725076675415, 0.9945654273033142, -0.9094716906547546, 1.570363163948059, -0.6997745633125305, 0.529967188835144, 0.24153712391853333, -0.8265263438224792, 0.0021335927303880453, -0.15234163403511047, 0.904565691947937, 0.2967112362384796, 0.19155222177505493, -0.18588177859783173, -0.012442436069250107, 0.9105412364006042, 1.3176414966583252, 1.9352880716323853, 0.8793143033981323, -1.172385573387146, -0.025708476081490517, 0.9331624507904053, -0.5878949761390686, -2.2424561977386475, -0.9296038150787354, -0.6396062970161438, 0.5167814493179321, 0.08008309453725815, 0.40427765250205994, 1.9991588592529297, -0.8658728003501892, 2.9606800079345703, 0.4445692300796509, -0.0880180299282074, 0.45471855998039246, 2.2179813385009766, 0.6083512902259827, 0.15263667702674866, -0.4400239884853363, 0.9285866618156433, 0.6063423752784729, 0.32576707005500793, 0.8456603288650513, -0.9436041712760925, 1.5473194122314453, -0.5863144993782043, 0.10135918855667114, -0.13753247261047363, 1.5480661392211914, -0.03281643986701965, 0.7385403513908386, -1.224937081336975, 0.6707420945167542, 0.7011727690696716, 0.5200982689857483, -0.7226338982582092, 0.8291862607002258, -0.656136691570282, 1.0820043087005615, 0.6561643481254578, -0.4382818937301636, 0.5434210896492004, -0.42912888526916504, 0.3395325541496277, 0.9346897006034851, 0.8348023891448975, 1.8098446130752563, 0.18057268857955933, -0.3217594623565674, 1.001632809638977, -1.185756802558899, -0.07941083610057831, 0.8566346764564514, -2.5462260246276855, -0.3596416115760803, 0.5609908699989319, -1.4020746946334839, -0.8446338176727295, 0.4613274931907654, 1.512474536895752, -0.37059450149536133, 1.0883400440216064, 0.7431336045265198, 1.073506236076355, 2.8195252418518066, 1.3228470087051392, -0.09078636020421982, 2.0910074710845947, 0.35753747820854187, -0.5059247016906738, -0.5073165893554688, -0.5000197291374207, 0.799071192741394, 0.3874348998069763, -0.2517491579055786, -0.143586665391922, -1.0990723371505737, -0.1001681238412857, -0.5904521942138672, 0.3001253306865692, -0.9450423121452332, -1.2142568826675415, 2.4710075855255127, -2.0090348720550537, 0.6701574325561523, 0.8592889308929443, 1.9335740804672241, 0.08861175924539566, 0.3671022951602936, 0.006214480847120285, -0.48704060912132263, 0.5136674642562866, -0.9636671543121338, 0.12232308089733124, 0.37606433033943176, 0.06162211298942566, -0.32094132900238037, 0.2570820748806, 0.8785109519958496, 0.9662690162658691, -0.08439627289772034, 0.6972751021385193, 0.09040942788124084, 1.0936774015426636, -1.6616121530532837, -1.6822330951690674, -0.3282739520072937, 0.5441091656684875, -1.062262773513794, 0.7580203413963318, 0.33015963435173035, -0.8243008852005005, 0.20227935910224915, -0.9766656756401062, 0.8829509019851685, -1.182879090309143, -0.25270381569862366, -2.0735671520233154, 1.1131576299667358, -0.5663306713104248, 1.0181771516799927, -1.909496784210205, -0.6685234904289246, -0.5003604888916016, -0.9473716020584106, -0.19994503259658813, 0.46943584084510803, 0.35862359404563904, 0.7726501226425171, 0.9411898255348206, 0.2320338785648346, -0.5610285401344299, -0.7894254922866821, 0.21415890753269196, 0.7323812246322632, -1.5680238008499146, 0.9457299113273621, -0.9601555466651917, -1.4486640691757202, -0.9529770016670227, -0.6143171787261963, -0.6929810643196106, 1.442703127861023, 1.128561019897461, 2.6602072715759277, 0.6610678434371948, -0.5976714491844177, -0.19789913296699524, -1.80319344997406, 2.1221702098846436, -0.49061673879623413, -0.2010892927646637, -0.5346787571907043, 0.42175063490867615, -0.5003705620765686, 0.30405980348587036, 0.5061033964157104, -1.6461387872695923, 1.43382728099823, 0.1558489352464676, -0.5978531241416931, -2.0185837745666504, 1.8785853385925293, -1.5936254262924194, -0.14027252793312073, -0.4854196310043335, -1.001355528831482, 0.13588695228099823, 2.5673482418060303, -1.1838010549545288, -1.4470815658569336, 1.4137537479400635, 1.1089129447937012, 1.6938385963439941, 2.6640875339508057, -0.8526116013526917, -1.3071821928024292, -0.18957379460334778, 0.19405274093151093, -0.5854132175445557, 0.12388770282268524, 0.7457728385925293, -0.594135582447052, -2.455012559890747, 0.18119266629219055, 0.9419927597045898, 0.16217973828315735, 0.1764054298400879, -0.17497742176055908, 0.9898025393486023, 0.4678914248943329, 0.28044605255126953, 0.7683963179588318, -0.7320170402526855, -0.26356497406959534, -1.9797364473342896, -0.9883926510810852, -0.8303028345108032, -1.0746033191680908, 1.095706582069397, 0.8481001853942871, 0.6629970073699951, 0.9967660307884216, -0.8162829875946045, -0.8347082138061523, -0.8440839052200317, -0.5391891598701477, -0.7696547508239746, 0.5659636855125427, -0.8758617639541626, 0.12770822644233704, -0.2743717133998871, -1.0359687805175781, -0.6611372232437134, 0.1579454392194748, -0.9183281660079956, 0.053671080619096756, 1.9305577278137207, -0.16789735853672028, 0.7242594957351685, -0.41553425788879395, 0.5547993779182434, 0.5720669627189636, 1.514214038848877, -1.4670721292495728, 1.0836209058761597, -0.3481663763523102, -0.12697146832942963, -0.659672737121582, 1.3276159763336182, 0.2892094552516937, -0.26424741744995117, 1.6018879413604736, -0.8957098126411438, 0.3148784339427948, -0.10165777057409286, 1.2940551042556763, -0.03373812511563301, -1.7703114748001099, 0.1569141447544098, 0.3043174743652344, 0.5190110206604004, -0.6928688287734985, 0.9967734813690186, 1.448112964630127, -2.0063555240631104, 1.0536799430847168, -0.9994997382164001, -1.0969377756118774, 0.6673069596290588, -0.696314811706543, -1.1401697397232056, 0.4977371394634247, -0.12413648515939713, -0.8070380687713623, 0.2503891885280609, 0.9868538975715637, 0.3235602080821991, 0.38812121748924255, -0.18514788150787354, 0.24471616744995117, 0.17074312269687653, -0.13254979252815247, 0.07216476649045944, -0.8453009128570557, 0.4617583155632019, 0.6871811151504517, -0.6668879985809326, -2.185311794281006, 0.008599738590419292], [-1.493489384651184, 0.7699865698814392, 0.7435104250907898, -0.1830364167690277, 1.9545552730560303, 0.2112094759941101, -0.901270866394043, -0.7762282490730286, 1.1243888139724731, 0.2835719585418701, 0.7056217193603516, 0.8315864205360413, -1.5621788501739502, -0.7480610609054565, -1.693926215171814, -0.6030905246734619, 1.2177844047546387, -0.8505041003227234, -1.5419223308563232, 0.3411305844783783, -0.14326563477516174, -1.6288748979568481, -0.9203441739082336, 0.6575906872749329, 1.8187377452850342, -0.8653781414031982, -0.4914206266403198, -1.017068862915039, -1.666867733001709, -0.3560372591018677, -0.537135899066925, 1.3395103216171265, 1.3744361400604248, 1.403574824333191, -0.026227938011288643, 0.1554727554321289, 0.7093268036842346, -1.4378371238708496, 0.46006831526756287, -1.7312180995941162, 2.3183040618896484, -0.7175076603889465, -1.0682525634765625, -1.061375379562378, 2.044614553451538, -2.175346851348877, 0.4196539521217346, 0.15334023535251617, -1.072135090827942, 0.4807693064212799, -0.8477286696434021, -2.410083532333374, 0.07673393189907074, 0.048940226435661316, -0.008067440241575241, -0.33783286809921265, 0.37185317277908325, -0.3404683768749237, -0.5435217022895813, 0.09051046520471573, -0.22874517738819122, 0.7256215214729309, -1.3890583515167236, 0.1301589012145996, -0.9495435953140259, 1.9871705770492554, 1.2646547555923462, 0.008150790818035603, 0.00020468827278818935, -0.4118349254131317, 0.4783501923084259, -2.2526626586914062, 0.4482199251651764, -0.7357569336891174, -0.6913435459136963, 0.0813361257314682, 0.5974904298782349, -0.16105994582176208, -0.09433701634407043, -0.571452260017395, 0.23534506559371948, 0.16210782527923584, -0.5878626704216003, -2.626683473587036, -0.6484237909317017, -0.5561259388923645, 0.22895808517932892, -2.016205072402954, 0.9651763439178467, 0.09098512679338455, -0.10824920237064362, -0.24591174721717834, -0.2821553647518158, 1.1326045989990234, 0.2411128282546997, 0.6215356588363647, 0.20622247457504272, 1.076447606086731, 0.8128877878189087, -0.04248754307627678, -1.2478532791137695, -1.4260928630828857, -0.39753177762031555, 0.44844815135002136, -1.09132981300354, -0.035162243992090225, 0.01303610485047102, 0.6400902271270752, -1.6812059879302979, 0.3284306526184082, -0.03403878211975098, -0.6603977680206299, 0.29636716842651367, 0.47417134046554565, -0.9998849034309387, 0.7026425004005432, 1.3192557096481323, 0.7697863578796387, 0.2712938189506531, -1.1592838764190674, -0.2142614722251892, -1.0485490560531616, 1.328354835510254, -0.7292877435684204, -0.729941725730896, -0.8717387318611145, 1.0993402004241943, -2.2467122077941895, -1.9208778142929077, 0.9473846554756165, -0.8431141972541809, 0.8091538548469543, -1.2646485567092896, -0.13019220530986786, -0.01885988749563694, -1.0274075269699097, -0.3430056571960449, 0.16828766465187073, -1.995819330215454, 0.20030033588409424, -1.603341817855835, -1.3851749897003174, 0.6675287485122681, 0.29917341470718384, -0.05414685979485512, -0.26142969727516174, 0.8194025158882141, -0.34543198347091675, -0.4012904465198517, 0.24230849742889404, 1.3342522382736206, -1.4813041687011719, 0.040062516927719116, 0.8946067094802856, -0.5050053000450134, 1.6711162328720093, 0.8040866255760193, -0.23737318813800812, 0.8471360802650452, -0.33048221468925476, 0.5581831932067871, -0.484491765499115, -0.874657392501831, -0.017572924494743347, 1.1529854536056519, -1.184030294418335, -0.8982858657836914, -0.5908169150352478, -0.39229944348335266, -1.6942532062530518, 2.045926094055176, 1.4142094850540161, 0.6962876915931702, 0.030886046588420868, -0.6035121083259583, -1.4745713472366333, -0.9518744945526123, -0.4135909974575043, 1.6738498210906982, -0.1220332458615303, -2.1122031211853027, 0.9244466423988342, 1.6105812788009644, -1.5607097148895264, 0.11004751920700073, -1.9164906740188599, 0.35717687010765076, -0.7809235453605652, 0.9945238828659058, 2.2126107215881348, 0.42654097080230713, 0.7349363565444946, 0.9007065296173096, 0.18359680473804474, -0.8876488208770752, 0.7955489754676819, -1.3194332122802734, -0.6333754062652588, -0.7621389627456665, 0.91169673204422, 0.7328245639801025, -0.11971117556095123, -0.2952543795108795, 0.8761724829673767, 0.0850447490811348, -0.6333346366882324, 1.65322744846344, -0.8057100772857666, -0.4983125627040863, 0.6392369866371155, -0.3708690106868744, -0.8739640712738037, 0.28817883133888245, 0.40433117747306824, 1.1077566146850586, -0.4917471408843994, -0.938931941986084, -1.2989479303359985, 0.1499723643064499, -2.5452327728271484, 1.4991148710250854, 1.9180576801300049, 0.7408944368362427, 0.3592150807380676, 0.8495982885360718, 0.7692533731460571, 0.9284446835517883, 2.809494972229004, 1.0566990375518799, 1.828710913658142, -1.8333743810653687, -1.2354799509048462, -0.11338008940219879, 0.32217323780059814, -1.0388810634613037, 0.10172826051712036, -0.5365540385246277, -0.26810380816459656, 1.417986273765564, 0.9592219591140747, -1.8890671730041504, 1.5319113731384277, -1.483652949333191, 0.9129683971405029, 1.6617960929870605, -0.2291887253522873, 1.2208607196807861, -0.2660186290740967, -0.42231276631355286, 1.1083924770355225, 1.2411324977874756, 1.9701474905014038, 0.6504252552986145, 0.4349309802055359, 1.7851331233978271, 0.4692021608352661, -0.07438312470912933, -1.3409457206726074, -0.40018463134765625, 0.7200965881347656, 0.5175547003746033, -0.016933731734752655, 0.3486044108867645, 0.8667613863945007, 0.37337008118629456, 2.821164846420288, -0.273450642824173, 0.12067101150751114, -1.4628009796142578, -0.6053606867790222, -0.35904207825660706, -1.9925258159637451, -0.23978528380393982, -1.2782329320907593, 0.14391040802001953, 0.8281484246253967, -1.7540031671524048, 1.7821704149246216, -0.8043180704116821, -2.196040391921997, 0.6970720887184143, 0.4172138571739197, -0.31554001569747925, -0.13057301938533783, -0.713322103023529, 0.6605221033096313, 1.3058452606201172, -0.07289616763591766, -0.9463358521461487, -1.5867429971694946, -0.8625285029411316, -0.10554081201553345, -0.7009001970291138, 0.1355818808078766, -2.1657252311706543, 0.15464557707309723, -0.6137580275535583, -0.7230876684188843, -1.0906522274017334, 0.1605888456106186, -1.9388757944107056, 0.8338426351547241, -2.9039459228515625, 0.48792359232902527, -1.6507266759872437, 2.132049798965454, 0.7687105536460876, -1.0651715993881226, -0.9067789316177368, 0.12368626892566681, 0.15153473615646362, -1.8847166299819946, -1.075960397720337, -0.0649389922618866, -1.2577608823776245, 0.17889876663684845, 0.16761387884616852, 0.6522641777992249, 0.9790796637535095, 1.0402629375457764, 0.5215590000152588, 1.1406238079071045, 0.04497573897242546, 1.5251283645629883, 1.5456429719924927, 0.4141193628311157, -0.9558109641075134, 0.9334359765052795, -0.5293520092964172, -1.7781864404678345, -1.887709140777588, 0.6222005486488342, -0.3821065425872803, 0.705815315246582, -0.33817437291145325, -0.8683392405509949, -0.006483964156359434, -0.7220585942268372, 1.059748649597168, -0.7281767129898071, 1.2500901222229004, -1.526569128036499, -0.5035786628723145, 1.3814494609832764, -2.076786518096924, -0.7246597409248352, 0.5563867688179016, 0.14149592816829681, -0.4181175231933594, -0.5366397500038147, 0.2126011848449707, 0.4173889458179474, 0.5059442520141602, -0.23193849623203278, 1.5005146265029907, 0.02079886570572853, 2.293870687484741, 0.6203740835189819, -1.3352445363998413, 1.0929713249206543, 1.2111270427703857, -1.8922609090805054, 0.6701620817184448, -0.002188291633501649, 0.508678674697876, 0.3704841136932373, -2.0452370643615723, 0.1497177630662918, -0.007161818910390139, 0.30498430132865906, -2.0537965297698975, -0.8636271357536316, 0.6331591010093689, 0.44547441601753235, 0.8982406854629517, 0.26203349232673645, 0.4048295319080353, -0.28373685479164124, 0.6035714149475098, 0.04940876364707947, -0.169233039021492, -0.04647725448012352, -1.0245496034622192, -0.08503448963165283, 0.3005157709121704, -1.1057974100112915, 0.6621257066726685, -2.4757649898529053, -1.2833606004714966, -1.886117696762085, -1.7520086765289307, -0.08889158070087433, -0.19755452871322632, -0.0353686697781086, 2.224107265472412, 1.062366247177124, 0.4917026162147522, -0.5747402906417847, 0.8308268785476685, 0.25399863719940186, 0.9671908617019653, 0.1063634380698204, 0.7980181574821472, 0.65423983335495, 1.5924423933029175, -0.7330767512321472, -0.7289740443229675, -0.9567340016365051, -0.4607290029525757, -0.6646184325218201, 1.0114153623580933, 0.9030183553695679, 1.185011863708496, -0.12157712131738663, -0.4572254419326782, 0.15506026148796082, -0.9451138973236084, -0.8466536998748779, 0.9136212468147278, 1.0417919158935547, -1.4677022695541382, -1.5232516527175903, -1.055704116821289, 0.20897941291332245, -1.1430473327636719, 1.390288233757019, -0.8135371208190918, -0.04394308477640152, 1.3642224073410034, 0.12502659857273102, -0.020196324214339256, 1.2128901481628418, 1.6923798322677612, 0.9757890105247498, 0.7455613613128662, -0.8155262470245361, 0.31744956970214844, 0.9214207530021667, 0.392598032951355, -0.8479456305503845, -0.5873918533325195, -0.16181106865406036, 0.6369373202323914, -0.19370010495185852, 0.9370793104171753, 0.45740604400634766, 0.5619433522224426, 1.176461935043335, -1.0963455438613892, -1.3287075757980347, 0.1528303474187851, 0.7749025821685791, -0.3883020579814911, 0.9859067797660828, 0.8444326519966125, 0.18748481571674347, -1.163848638534546, 0.7583798170089722, -1.6084706783294678, 1.0929160118103027, -2.5546109676361084, -0.21524757146835327, 1.3349233865737915, 1.3797898292541504, -0.11737686395645142, -0.26931679248809814, 1.96188485622406, 0.07412762939929962, -0.5319714546203613, -0.23599854111671448, -0.6618682146072388, 0.1816510409116745, -0.9545422792434692, 0.44339656829833984, 0.22356440126895905, -0.48713716864585876, -0.061624422669410706, 0.8305146098136902, -1.0297218561172485, -1.3705579042434692, -0.31032201647758484, -1.461197018623352, -0.39861786365509033, -1.6036059856414795, 1.0287902355194092, -0.22463101148605347, 1.0422028303146362, -1.2624149322509766, -0.14213407039642334, -0.5543693900108337, 0.05231666564941406, 0.4053412973880768, -0.31336474418640137, 1.135387659072876, -0.079157255589962, 0.4203002154827118, -0.8210000395774841, 0.0326855443418026, -0.03409748151898384, 0.1118946224451065, 1.4667439460754395, -0.20932498574256897, 0.6161829829216003, -0.982720136642456, 1.1089931726455688, -0.3294925391674042, 1.555894374847412, -1.5622094869613647, 2.416956901550293, -0.7490865588188171, -0.695155680179596, 0.37540319561958313, 0.08920168876647949, 2.092174530029297, 0.7212395071983337, -1.1286979913711548, -0.5395561456680298, 1.1217683553695679, -0.028759675100445747, 2.0230233669281006, -1.249712347984314, 0.23343950510025024, 0.36198610067367554, 0.12038219720125198, -0.7672551274299622, 0.6191188097000122, -0.6735309958457947, 1.2564024925231934, 0.9812533259391785, 1.3187565803527832, 1.6800305843353271, 1.404904842376709, -0.46444323658943176, -0.042042095214128494, -0.11445583403110504, 0.5931118130683899, -0.23288168013095856, 0.7351273894309998, 1.239824891090393, 1.763003945350647, -0.3423033058643341, 0.4505971074104309, 0.023044750094413757, -1.0184606313705444, -0.02616070583462715, -1.451740026473999, 0.6186848878860474, -1.1986089944839478, -0.41785064339637756, -1.183711051940918, -0.20946337282657623, 0.4022998809814453, -1.4084969758987427, -0.9231041669845581, -1.1349130868911743, 0.8127288818359375, 0.5058517456054688, 0.23173102736473083, 0.5058117508888245, 0.2271561473608017, 0.10862795263528824, 0.2081882506608963, -0.38240694999694824, 0.9797077178955078, -0.17559710144996643, 0.629811704158783, 2.753175973892212, 0.13445711135864258, -0.2730468213558197, -0.10656751692295074, -0.7874424457550049, 1.0809582471847534, -0.21058236062526703, 0.5622019171714783, 0.9317538142204285, 1.2444884777069092, 0.45416492223739624, 0.7233463525772095, 0.34792888164520264, -0.12998521327972412, -0.6986008882522583, 2.4917454719543457, 1.5441582202911377, 0.5388197302818298, -0.3570792078971863, 0.9151337742805481, -0.766082227230072, -0.5935515761375427, 1.8813380002975464, -1.1700531244277954, 0.13316652178764343, -0.5978684425354004, 0.42211100459098816, 0.43414077162742615, 0.6022862792015076, 0.4107019603252411, 0.09647142887115479, 0.06327860802412033, 0.4781463146209717, -0.33437982201576233, -0.22930747270584106, 0.8321354389190674, 0.22501051425933838, -2.0321052074432373, -0.359814316034317, -0.23227716982364655, 1.5162616968154907, 0.8331934809684753, -0.6493091583251953, -0.26675984263420105, -0.47634410858154297, -0.6354857087135315, -1.9966275691986084, -1.0189250707626343, -0.2688467800617218, 0.9304379224777222, 0.27888941764831543, 0.24912752211093903, -1.3939661979675293, -0.5793097615242004, -0.7015889883041382, -1.0926076173782349, -1.9865572452545166, 0.7450770735740662, 0.3087102472782135, 0.47138506174087524, -0.7820126414299011, 0.6238383054733276, -1.0517690181732178, 1.5954378843307495, -1.4807277917861938, -1.184245228767395, -0.17195236682891846, 0.27568870782852173, -0.633080005645752, -0.34703928232192993, 0.6404446363449097, 0.13560476899147034, 1.596130609512329, 0.46917954087257385, 2.0456528663635254, 0.9426282048225403, 0.8487560153007507, -0.014491351321339607, -0.27070102095603943, 0.07001350820064545, -0.1342456340789795, 0.13964800536632538, 1.3295702934265137, -1.0017826557159424, 0.2986297905445099, -0.4876709282398224, 1.1385201215744019, 1.0995889902114868, 0.2998640537261963, -0.07681675255298615, -0.36658763885498047, 1.09267258644104, -1.2739149332046509, 0.9505302906036377, -1.0651594400405884, -0.5841519236564636, -0.30180567502975464, -1.1699562072753906, -0.18257832527160645, -2.289296865463257, -0.08403462171554565, -0.8118873238563538, -1.5776439905166626, -0.515332818031311, -0.6373212933540344, 0.7519369721412659, 0.42030248045921326, -1.3066085577011108, 0.36245012283325195, 1.1240743398666382, 0.22599327564239502, -0.956470251083374, 0.9031192064285278, 0.3201884329319, -0.39504602551460266, 0.6647261381149292, 0.6566535234451294, -0.7259106040000916, -0.8606985211372375, -0.5765003561973572, -0.21683771908283234, -1.0012142658233643, 1.2226264476776123, 1.3454463481903076, -0.023268956691026688, 0.6396045684814453, 0.10332319885492325, -1.6107416152954102, -0.4802747070789337, 0.37583738565444946, -0.6290981769561768, -0.7216540575027466, -0.49133574962615967, 1.2312846183776855, 0.7987872362136841, -0.7819370627403259, 0.28009361028671265, -0.9250196218490601, -1.413291096687317, -0.7708365321159363, 0.7418879270553589, 0.2701921761035919, -1.0713870525360107, -0.44688743352890015, -0.4051293432712555, -0.36385875940322876, -2.117492437362671, -1.2144701480865479, 0.22526337206363678, -1.6622276306152344, -0.1918768435716629, -1.2064381837844849, -0.4720703065395355, -0.8122959733009338, 0.3209785521030426, -0.5577833652496338, 0.5143343210220337, 1.5510387420654297, -1.0854755640029907, 0.7284680604934692, 2.4535815715789795, -0.8473747968673706, -0.4100930392742157, 0.8367552161216736, -0.2903524935245514, 0.5894171595573425, 0.4841793179512024, 1.2157343626022339, 0.5583946704864502, -0.19129535555839539, -0.7770150899887085, -0.279875785112381, 0.08097447454929352, -0.008520434610545635, 0.9124404191970825, -0.31667450070381165, 2.4993655681610107, -1.409132480621338, -0.2718685567378998, -0.9816754460334778, 0.10824626684188843, -3.1800763607025146, -1.404210090637207, 0.7989984750747681, -0.4653405547142029, -0.6312037110328674, 2.1430578231811523, 0.34196141362190247, -0.48434942960739136, -1.0432387590408325, -0.1267760694026947, 0.9630640745162964, 0.23959562182426453, 1.131976842880249, -1.6066815853118896, 1.2164667844772339, 1.1641443967819214, 1.6662126779556274, -0.5857771039009094, -0.06361802667379379, 0.9814645648002625, -0.9472663402557373, -1.139534831047058, 2.0291144847869873, -0.5040490031242371, -1.7566665410995483, 0.7635293006896973, 1.871611475944519, 0.6051007509231567, 1.1478030681610107, 1.262546181678772, -2.078763484954834, -0.45872777700424194, 0.9084813594818115, 2.4960689544677734, -0.8104525208473206, 0.3881080746650696, -0.6880325078964233, -0.31661900877952576, 0.43423765897750854, 1.0986469984054565, 0.15386256575584412, 0.47742828726768494, 0.3716953992843628, 1.7924944162368774, -0.6106293201446533, 0.054244618862867355, -0.8007702231407166, 0.49808624386787415, -1.0140295028686523, -0.9976293444633484, 0.5021523833274841, 0.1654927134513855, -1.1374589204788208, 0.6539638638496399, -0.18904642760753632, -0.2264266312122345, -2.6986865997314453, 0.9679571390151978, -2.061170816421509, 0.23656003177165985, 0.3530842065811157, -0.0676933154463768, 0.9954535365104675, 0.11572439223527908, 0.07371301203966141, -0.9400054812431335, -0.38212326169013977, 0.31958696246147156, 0.6769430637359619, -0.5131049156188965, -1.6598261594772339, 0.5544227361679077, -1.423357367515564, 0.9607462286949158, 0.9708617329597473, -0.2694104015827179, -0.33886104822158813, -0.03669627383351326, 0.5804402828216553, 0.41729238629341125, 2.492802143096924, 0.47623956203460693, 1.4532581567764282, -0.5510092973709106, 0.3373674750328064, -1.0891746282577515, -1.3050225973129272, -0.7050225138664246, 0.6822479963302612, 0.3192006051540375, -0.912219762802124, 1.3439756631851196, -0.4986227750778198, 0.10968471318483353, -1.0755220651626587, -0.07862124592065811, 0.7931610941886902, 0.513272225856781, 0.6500566601753235, 0.5974892973899841, 0.24033650755882263, -0.32801488041877747, -1.5976542234420776, -0.6043562293052673, -0.693368136882782, 0.4071073532104492, -1.4311692714691162, 1.2796378135681152, 0.5075769424438477, 0.9700683951377869, -1.1086310148239136, -0.24617739021778107, 0.6050719022750854, 1.504089593887329, -1.5878397226333618, -0.1963384598493576, 1.0093193054199219, 0.493074506521225, 1.0159316062927246, 1.811023473739624, 1.0599561929702759, -0.2575477659702301, -0.6761121153831482, -1.7259511947631836, 1.0662559270858765, -0.6180559992790222, -0.05399186536669731, 1.741335153579712, 0.9513248205184937, 0.16320733726024628, 0.3016291856765747, -1.1907397508621216, -1.1439777612686157, 1.5332226753234863, 1.7411772012710571, 1.182844638824463, -0.2812110185623169, -0.06579073518514633, 0.9538577198982239, -0.7560322284698486, -0.08487094938755035, -1.1398717164993286, 1.3972111940383911, 0.6290420889854431, -0.7345873117446899, -1.0775809288024902, 0.19881296157836914, -0.3687478005886078, 1.2004586458206177, 0.6192879676818848, 2.267350912094116, 0.6367607712745667, -1.2896077632904053, 1.1735745668411255, -0.27303066849708557, -1.9323939085006714, 0.3880487382411957, -0.36821818351745605, -1.4321995973587036, -1.2649413347244263, -0.6692224144935608, 1.5302809476852417, -1.1688733100891113, -1.2907989025115967, 1.7950836420059204, -0.5900830626487732, 0.19099271297454834, -0.49304044246673584, -0.5566667914390564, -0.26694777607917786, -1.4480831623077393, 0.20291276276111603, 0.5002825260162354, -1.1198300123214722, -0.13733159005641937, 1.3714436292648315, 0.7326717376708984, -0.5012429356575012, -0.9866378903388977, -0.7158316373825073, 1.1570384502410889, 1.1668754816055298, -0.44127726554870605, 0.457872211933136, 0.19899721443653107, -1.9205567836761475, -0.10063015669584274, 1.1999675035476685, -1.4028841257095337, -1.3357809782028198, 2.077045202255249, 0.6806079149246216, 0.5101720094680786, -1.071170687675476, 0.6736631989479065, -1.1384735107421875, 0.7715258002281189, 1.1335862874984741, -1.6953001022338867, 0.6477588415145874, 1.247199296951294, -0.3834381103515625, 2.36740779876709, -0.7070249319076538, 0.16021080315113068, -0.25282278656959534, -1.1062415838241577, -1.1845742464065552, 0.9729499220848083, 0.5838181972503662, 0.37221837043762207, -1.4281257390975952, -0.8666149377822876, -0.5626477599143982, 0.1695018708705902, 0.30480584502220154, 1.600791573524475, -1.6458873748779297, -0.32050496339797974, 1.2533237934112549, -1.5152387619018555, -1.2521642446517944, -2.634239435195923, 0.13974516093730927, 0.45479822158813477, -0.24569609761238098, 0.4796648919582367, -1.844468593597412, 0.7025864124298096, -1.2120496034622192, -1.6417745351791382, -0.2309216856956482, 0.5153957009315491, 0.7846564650535583, -1.8463619947433472, 1.0343894958496094, -2.041365623474121, -0.547144889831543, 0.4394613802433014, 1.5144758224487305, -0.4913983643054962, -0.48398250341415405, -0.19253164529800415, -1.024416208267212, 0.6723240613937378, -0.7002840042114258, -1.700168490409851, -1.6092171669006348, 2.188170909881592, 1.0509635210037231, -0.7948039174079895, 1.4575519561767578, 1.3850845098495483, -0.6537569165229797, 0.7300966382026672, -0.6218822598457336, 0.16978318989276886, 0.01392899639904499, -0.21951857209205627, 0.3720462918281555, -1.3676601648330688, -0.7158098816871643, 1.2412265539169312, 0.7626804113388062, 0.5307114720344543, 0.2320374846458435, 1.2614065408706665, 0.5554795861244202, 0.5212026238441467, 0.6122252941131592, 0.18567001819610596, -0.6243292093276978, 1.0475934743881226, 1.8422600030899048, -0.2533878684043884, -0.5542013049125671, -0.5049380660057068], [-1.3863551616668701, -0.08494902402162552, 0.3986261785030365, -0.8362893462181091, -0.2816890478134155, 0.3284136652946472, 0.682861328125, 1.2067902088165283, -0.13403384387493134, 0.0506451390683651, -1.401577353477478, 0.3939582407474518, -0.014103067107498646, 1.8707884550094604, -0.362691193819046, -1.254330039024353, -0.15210269391536713, 0.705350935459137, 1.142961025238037, -0.8618097901344299, -1.8533297777175903, 0.8641612529754639, -1.270005226135254, 0.32252559065818787, -0.4950106739997864, 0.5281383991241455, 1.5108548402786255, 0.8417662978172302, 0.5197869539260864, -0.3897216320037842, 0.22605396807193756, 0.7812355160713196, 0.4564597010612488, 1.2271844148635864, 0.07094139605760574, -0.691202700138092, 0.33147135376930237, -1.3991786241531372, -0.13309797644615173, 1.363770604133606, 0.5770416259765625, -0.4541236460208893, -0.38133737444877625, 0.7634260058403015, 1.074996829032898, 0.015108385123312473, -1.6166949272155762, -0.7895949482917786, -0.3326473534107208, 0.6900225281715393, 0.4475952684879303, 1.269783854484558, 0.7257885336875916, 1.1291389465332031, -1.7866319417953491, 0.8954697251319885, -1.8120187520980835, -0.07391458749771118, -1.3750654458999634, 0.20493966341018677, 1.095443844795227, 0.5493240356445312, 0.9751361012458801, -1.1726009845733643, 0.8908416628837585, 0.4125952124595642, -0.4133070707321167, -0.3924659490585327, -1.7109479904174805, -0.5345171689987183, 0.9303893446922302, 1.0566258430480957, -0.3831483721733093, -1.0543371438980103, 0.5175497531890869, -0.9654116034507751, 1.618586778640747, -0.7271664142608643, -0.3140614926815033, 0.0869932621717453, 0.7466646432876587, 2.1406309604644775, -1.5043023824691772, -0.20343706011772156, 0.27063629031181335, -0.07161428034305573, -2.083494186401367, 0.4452849328517914, -1.4375325441360474, -0.6727197170257568, -2.190769672393799, -0.829912006855011, -0.8547332286834717, -1.1149388551712036, -0.24702011048793793, -0.40169423818588257, 0.3175455331802368, 0.7145767211914062, 1.3104010820388794, -1.6260335445404053, 0.32850462198257446, -1.0718817710876465, -0.8815800547599792, 0.19448025524616241, 0.27740901708602905, -0.2110167145729065, 1.3289451599121094, 1.3529425859451294, 0.6587170958518982, 1.9379022121429443, -0.017954468727111816, -1.1206340789794922, 0.24767859280109406, 1.5025131702423096, 0.359966903924942, 0.08367173373699188, -0.7224400639533997, 2.0913753509521484, 1.062697410583496, -0.11166233569383621, 1.2582807540893555, -0.775243878364563, 1.2992026805877686, 1.0035816431045532, 2.2751755714416504, -0.9765379428863525, 0.39993026852607727, 0.13150663673877716, -0.7805538773536682, 0.0865936204791069, 0.4531170427799225, -0.20102578401565552, 0.9549872875213623, 0.4693964719772339, -1.1337928771972656, -0.22284561395645142, -1.259244680404663, -1.4105281829833984, 0.3282788097858429, 1.6447826623916626, -1.6019887924194336, 0.9427957534790039, -0.4442775249481201, 1.2068450450897217, -0.2624655067920685, 0.8073638081550598, -0.7199405431747437, 0.12156613171100616, 0.1448221504688263, -0.12615875899791718, 1.3551864624023438, 0.6372815370559692, 1.1596391201019287, -0.4754289984703064, 0.1708589494228363, -1.8045355081558228, 0.7597404718399048, -1.0717734098434448, 0.05250829458236694, -0.23908843100070953, -0.8691008687019348, 0.30873870849609375, -0.7220460772514343, -0.5913839936256409, -1.5356088876724243, -1.4077945947647095, 2.125732421875, 0.09135835617780685, -0.5228103995323181, 1.7922163009643555, -1.230963945388794, -0.32859888672828674, 0.25157374143600464, -0.7957258224487305, -0.2594488561153412, 1.1266471147537231, -1.3599400520324707, -0.98233562707901, -0.26494309306144714, 1.3847404718399048, -0.9331214427947998, -0.4792065918445587, 1.2779878377914429, -0.774763286113739, -0.5531090497970581, 1.200082778930664, -1.3607308864593506, 1.3013023138046265, 0.1871819943189621, -0.9070164561271667, 0.9453690052032471, 3.3400394916534424, -0.5811322927474976, -0.734069287776947, -1.3455450534820557, 0.3133363425731659, -0.7171981334686279, 0.09216660261154175, 1.9525141716003418, -0.23914070427417755, 0.0030477559193968773, -0.36831530928611755, 0.3027679920196533, 0.6357492208480835, 1.6623793840408325, -1.6121491193771362, 0.8661184906959534, 2.111349582672119, -1.0961496829986572, 0.6558209657669067, -1.3000342845916748, -0.3067930340766907, -0.6311210989952087, -0.4500497281551361, 0.1507972925901413, -0.8540350794792175, 0.45350348949432373, 1.479784607887268, -0.8169359564781189, -0.8029614090919495, -0.8455731868743896, -0.595933735370636, 0.561569094657898, 0.28667977452278137, 0.27912548184394836, 0.23643575608730316, 0.7768625617027283, -0.9663971066474915, 0.7089399695396423, 1.743752121925354, -1.5849685668945312, 0.6317651867866516, -0.2301880419254303, -0.46534302830696106, -0.46334967017173767, -0.7324056029319763, -0.15980519354343414, -0.26343977451324463, 0.08193822205066681, -0.7498137354850769, -0.7582648396492004, -0.8117042779922485, 0.0052368915639817715, -0.7650811076164246, 0.6663822531700134, -0.4635738730430603, -0.11924310028553009, 1.6811749935150146, -0.9154961109161377, -1.1748017072677612, -0.8864997029304504, -1.3961296081542969, -0.21890470385551453, -0.9381848573684692, 0.09453898668289185, -0.03450137749314308, 0.8791918754577637, 1.365882158279419, -1.0509989261627197, -1.0763750076293945, -0.209380641579628, -0.9818755388259888, -1.851737141609192, -0.7743707895278931, 1.2161498069763184, -0.21630893647670746, 0.46713200211524963, -0.5806311368942261, -0.01933935657143593, 0.4304262399673462, -0.9136964678764343, -0.6603646874427795, -1.4310040473937988, 0.0812060758471489, -0.03596358001232147, 0.03296639397740364, -0.40082868933677673, 0.452422559261322, 1.834548830986023, -0.13097819685935974, -0.1274794191122055, -0.6625131964683533, -1.739866852760315, 1.5340303182601929, -0.17508472502231598, -0.2398919314146042, 1.9940515756607056, -2.4426181316375732, -0.6752943396568298, -1.1900749206542969, 0.23695136606693268, -1.1409512758255005, -0.5596939325332642, -0.9225749373435974, -0.6830368041992188, 0.0626610815525055, -0.20217721164226532, -0.9710158109664917, 0.49874913692474365, 0.7371873259544373, -0.843341588973999, 1.408150315284729, -0.6331398487091064, -0.3610307574272156, -0.0798100158572197, 0.5835913419723511, -0.9264145493507385, -1.1326138973236084, 0.8469846844673157, 0.23599477112293243, -0.29189029335975647, 0.7883127331733704, 0.5875638723373413, 0.43119582533836365, 2.105991840362549, 0.7883203625679016, 1.28643000125885, -2.0819766521453857, 0.005572321359068155, 0.975430965423584, -0.6321245431900024, -0.5596290826797485, -0.17918938398361206, -1.1439251899719238, 1.6757187843322754, 0.12575416266918182, -1.7165651321411133, -0.08536629378795624, 3.196559190750122, -0.3248293697834015, -0.8200662732124329, -0.008096401579678059, -0.9109417796134949, -0.29133108258247375, 0.37486281991004944, -0.9726778268814087, 0.6776660680770874, 1.218441367149353, -2.2359795570373535, 2.1032800674438477, -0.3367924392223358, -0.9399946331977844, 0.5174070596694946, 1.6753768920898438, -0.571381688117981, 0.14736640453338623, -1.1409705877304077, -0.5576319694519043, -1.3967336416244507, 0.7068489193916321, -0.8969748616218567, -0.9097646474838257, -0.8688439130783081, 0.10316900908946991, 0.9264143109321594, 1.6713392734527588, -1.7513269186019897, 0.9800325036048889, -0.6780099272727966, -0.09500037133693695, 2.0806376934051514, -0.5630977749824524, 0.007096361368894577, 0.44845467805862427, 1.9826279878616333, 0.7398743033409119, 0.5199645757675171, -0.9048694968223572, -1.010879635810852, -0.7442020177841187, 0.45921018719673157, -1.3067841529846191, 0.27571359276771545, -0.3221043050289154, 1.1050692796707153, -1.4868104457855225, 0.1363886445760727, 0.5051811337471008, -0.4376145899295807, -0.5791187286376953, 0.4031369388103485, 0.930402934551239, 1.1558194160461426, -0.1772206723690033, 1.3763372898101807, 0.8281681537628174, 0.09627288579940796, 0.0021937466226518154, -0.8150002360343933, 0.39575543999671936, -1.6242859363555908, -0.41301411390304565, -0.5558292865753174, -1.5002684593200684, 1.3514280319213867, 1.4331012964248657, 1.1766258478164673, -1.2023171186447144, 0.012152111157774925, -2.175467014312744, 0.9470342397689819, -0.8947424292564392, 0.7440903782844543, -1.2260302305221558, 0.6859275102615356, 1.986175537109375, -0.33285823464393616, 0.7125266194343567, 0.6223520636558533, -0.5816906094551086, -0.01327283401042223, -0.5408207178115845, 0.0330825112760067, -0.8289221525192261, 1.1019642353057861, 0.6311970949172974, 0.5465930104255676, -1.1533087491989136, 0.7466702461242676, 0.42325565218925476, 1.6584724187850952, -0.15343602001667023, -0.47150835394859314, -0.23141801357269287, 0.2004004865884781, -0.08345992863178253, 2.573935031890869, -0.36630478501319885, -0.5906198024749756, -0.28516459465026855, 0.8891382217407227, 0.8800114393234253, 1.0237418413162231, -0.5747271180152893, -1.1781055927276611, 0.5760161876678467, 0.19601911306381226, 0.11957772821187973, -0.9492608904838562, -0.3073410391807556, -0.2667347490787506, 0.3989507853984833, 0.18031734228134155, -0.015098574571311474, -0.1562543660402298, 1.2410460710525513, 0.14519204199314117, 0.21788568794727325, 0.5107600092887878, -1.588416576385498, 0.31578946113586426, 0.7179799675941467, -0.2705598771572113, 0.7285195589065552, -0.5607624650001526, 2.228785276412964, 1.1974226236343384, 0.47755908966064453, 0.48395317792892456, 2.2790000438690186, -0.7713410258293152, 1.0214378833770752, -0.002861347747966647, -0.17994362115859985, -1.633361577987671, 0.37132537364959717, -1.0043842792510986, 0.5067120790481567, -0.3140908479690552, -0.35536137223243713, 1.1408283710479736, 0.1562492549419403, -0.2545261085033417, 0.800243079662323, -0.615599513053894, 1.0865516662597656, -0.8881731033325195, -1.2014641761779785, 0.3226153552532196, -0.2574833631515503, -1.4631397724151611, -0.34148329496383667, -0.42175132036209106, -0.3027729392051697, 1.0708163976669312, -1.9989992380142212, 2.224161148071289, -0.008639148436486721, 0.31745120882987976, 0.42250874638557434, 1.371561884880066, -1.9087947607040405, 1.4225218296051025, -1.4491794109344482, -1.0147678852081299, -0.08022966980934143, 1.9366668462753296, 1.54412043094635, -0.012916097417473793, -1.6320209503173828, -0.7946031093597412, 0.24238459765911102, 0.3020161986351013, -0.5995115041732788, 1.0655560493469238, 0.4982568919658661, 0.10004030168056488, -1.2447106838226318, -0.6201266646385193, -2.1429760456085205, -0.7346106767654419, 0.6480464935302734, 1.0772720575332642, -0.22859641909599304, 2.211223602294922, -1.1617515087127686, -0.8034927248954773, 0.007835481315851212, 0.5668140053749084, 1.8536179065704346, 0.6798449158668518, -0.5153530836105347, 0.4869332015514374, -1.3754993677139282, -0.8006990551948547, 0.7924977540969849, -1.4255907535552979, -0.09561484307050705, -0.2800734341144562, 1.0509544610977173, -0.281121164560318, 0.5653413534164429, -2.5837080478668213, 0.20875750482082367, -0.39949584007263184, 0.41605308651924133, -0.07518445700407028, -0.27354878187179565, 1.8870670795440674, -0.19945493340492249, -0.3845391869544983, -2.3015658855438232, -0.1565004587173462, 1.1098523139953613, 0.6877287030220032, -0.285149484872818, -0.5270105600357056, -0.5046571493148804, -0.41627857089042664, 0.22626878321170807, -1.2580660581588745, -1.1701710224151611, -0.6970760226249695, -0.605343222618103, 0.36178696155548096, 0.6924678087234497, 1.3710882663726807, -0.6504833698272705, -0.6939985752105713, -1.1603193283081055, 1.0840526819229126, -0.2579827904701233, -0.06807146221399307, -0.5119348764419556, 0.5774984359741211, -1.9851293563842773, -0.003911428619176149, -0.2860935926437378, -0.48618870973587036, 1.3267041444778442, -0.09248962253332138, 0.10996869206428528, -0.17226319015026093, -0.04576084390282631, -0.7042181491851807, -1.1281208992004395, -2.5950825214385986, 0.09393146634101868, -2.3295366764068604, 0.3948478400707245, 0.09184525907039642, -0.7629451751708984, -0.20034849643707275, 0.5931877493858337, -0.19559746980667114, 1.7620431184768677, 0.25522708892822266, 0.8299398422241211, -0.489651620388031, 2.0716679096221924, -0.3870512545108795, 0.44355249404907227, 1.4998608827590942, -2.106462240219116, -2.8187639713287354, 0.8597783446311951, 0.8396483659744263, 1.4609148502349854, 1.0509448051452637, 0.7226740121841431, 0.6363318562507629, -0.18907850980758667, 0.8351849913597107, -2.4874770641326904, 1.1397958993911743, -0.15914121270179749, 1.235007643699646, 1.888472557067871, 0.632305920124054, 0.48242390155792236, 0.3730774521827698, 1.967341661453247, 1.342393398284912, -0.1579868495464325, -0.5869417786598206, -1.6589514017105103, -1.4948043823242188, -0.10767176002264023, 0.8794261813163757, -2.032003879547119, 1.9640928506851196, -0.03163115680217743, -0.2766765356063843, -1.1593552827835083, -0.6097027659416199, -1.028635859489441, 1.0517972707748413, 0.6317842602729797, 0.6836491823196411, -0.5164566040039062, -0.2002246379852295, -0.180090069770813, -0.04263821244239807, -0.7362330555915833, -1.27926504611969, 2.247713804244995, 0.5107787847518921, 0.6999467015266418, 0.8736971616744995, -0.45723649859428406, 1.0537159442901611, 2.3709919452667236, -0.2776738405227661, -0.5496412515640259, -0.22708363831043243, 0.08774338662624359, -1.1905573606491089, -0.5783793926239014, -0.9883954524993896, 0.4198291301727295, 0.9343406558036804, 0.4790875017642975, -1.7456672191619873, 1.7510613203048706, 0.5113706588745117, 0.7564435005187988, 0.12521927058696747, 1.4473834037780762, -2.539529323577881, -0.6852181553840637, -0.40748316049575806, 2.1020495891571045, -1.3144092559814453, -0.28833481669425964, -1.4604148864746094, 0.6628717184066772, 2.1310932636260986, 0.5367332696914673, -0.7832691073417664, -0.6168217062950134, -1.3011234998703003, -1.55326509475708, 0.22214923799037933, -0.5551290512084961, 0.6934488415718079, 1.3737140893936157, -0.11786940693855286, -0.13154751062393188, -0.20044396817684174, 1.563621163368225, -0.6306368112564087, -2.0940046310424805, 0.6705331206321716, -1.0755935907363892, 0.22737130522727966, -0.57096928358078, 0.986790657043457, -1.4640047550201416, -0.4485590159893036, -0.9292453527450562, -0.41730377078056335, -0.01028379611670971, 0.6633533835411072, -0.5679831504821777, -0.5613314509391785, -0.4513561427593231, -1.4763904809951782, -1.163089632987976, -0.12993986904621124, -0.0024401431437581778, 0.5091124176979065, -1.6601043939590454, -1.5767537355422974, 1.077235221862793, 0.0007448708638548851, 1.7196779251098633, -2.7487406730651855, 0.8311361074447632, 1.1279712915420532, 1.1272304058074951, -0.7426691055297852, 0.05024158954620361, -0.4550975263118744, -0.9090333580970764, -0.3193676471710205, -2.1187920570373535, 0.3070715367794037, -0.3715808689594269, -0.1369306892156601, 2.2946672439575195, 1.4793810844421387, 0.750907301902771, -0.6963992118835449, -0.6099991202354431, -0.37490591406822205, -0.20204532146453857, 2.1166412830352783, -2.0939748287200928, 0.5805840492248535, 0.8269935846328735, 0.3704811632633209, 1.1259735822677612, 0.22516103088855743, -0.7393847703933716, 0.21164430677890778, 0.17869801819324493, 2.025546073913574, -0.7390552163124084, -1.5189285278320312, 0.2014320194721222, 0.6644372344017029, -0.582973837852478, -0.8100500702857971, 1.6103681325912476, 0.955447256565094, -0.913593590259552, 0.10310292989015579, 0.5385022163391113, 0.8187482953071594, 0.5706888437271118, -0.8572032451629639, 1.5778353214263916, -1.0446603298187256, 0.4523622989654541, 1.3928996324539185, 0.563543438911438, -0.6131278276443481, 0.0337621346116066, -0.3718522787094116, -1.0794365406036377, -1.039726734161377, -0.5603158473968506, 0.4670831561088562, -1.4243707656860352, -0.3533286154270172, -0.0008568362100049853, 1.0776863098144531, 0.47949427366256714, 1.82542884349823, -0.4725401997566223, 0.8962951302528381, 2.2019917964935303, 0.20160919427871704, -1.6737091541290283, -0.42925748229026794, 0.5979127287864685, 0.6836201548576355, -0.3397253751754761, 0.8926412463188171, 0.7787398099899292, -0.3000427782535553, 0.2968538999557495, 1.1026012897491455, -0.34458982944488525, 1.1651203632354736, 0.5426774024963379, 0.0626477375626564, 1.318672776222229, -0.24745406210422516, -1.9140015840530396, -1.1317681074142456, -0.7206161022186279, 0.6585699915885925, 1.1779009103775024, 0.851021409034729, -0.5739833116531372, 1.5221178531646729, -1.401520848274231, -0.6277072429656982, 1.7803376913070679, 1.816807508468628, -1.2807941436767578, -0.13411760330200195, 1.495238184928894, -0.1301613748073578, 0.8881925940513611, -0.03820141404867172, 0.012574452906847, -0.4692392647266388, -0.41828683018684387, -0.48871904611587524, 1.4995572566986084, 0.21017558872699738, -1.3300082683563232, 1.6349990367889404, 0.5023627877235413, -0.8893603086471558, -1.525733232498169, 0.3086661994457245, -0.8511583209037781, -0.17418743669986725, -1.4222010374069214, 0.14729179441928864, -0.25286099314689636, 0.15810218453407288, -0.672524094581604, -0.7822133302688599, -0.6926102042198181, 1.5520687103271484, -3.0868165493011475, 0.6782777905464172, -0.45077943801879883, 0.5031987428665161, -1.8160604238510132, 0.03356771916151047, 0.5142337679862976, 0.16380628943443298, -0.4711458384990692, 0.027686191722750664, -1.8619470596313477, -1.9756947755813599, 1.4241820573806763, 0.5525858402252197, 1.3247624635696411, -0.31160056591033936, 0.1418236345052719, 1.1547906398773193, -0.27205735445022583, -1.9973084926605225, -1.1164371967315674, 0.12482522428035736, -1.6017320156097412, 1.262195348739624, -0.20235973596572876, -0.18740184605121613, -0.9006941914558411, 0.4221022427082062, 0.9133251309394836, 1.2770729064941406, 0.2732168138027191, 0.5566272735595703, -3.0204594135284424, -0.05606193095445633, 0.7941935658454895, -2.4630134105682373, 0.683169960975647, 0.8577871322631836, -1.9135340452194214, -0.5733315944671631, -0.4658016860485077, 0.5055953860282898, -1.9309475421905518, 0.17825590074062347, -1.1207112073898315, -0.046811122447252274, 0.7271062731742859, 1.38233482837677, 1.4079545736312866, 1.125152349472046, -0.25915300846099854, 0.14489148557186127, -0.9256299138069153, -0.25021055340766907, -0.2552737593650818, 0.1534918248653412, -0.7115968465805054, 0.12047480791807175, 0.5297505855560303, -2.2712268829345703, -1.3368709087371826, -0.046278033405542374, 0.6043322086334229, -0.6466444730758667, -0.7876921892166138, -0.5990386009216309, -0.8196623921394348, 0.3972700834274292, -1.2845056056976318, -0.5475354194641113, 0.6689284443855286, -0.09986647218465805, 1.052770972251892, -1.1299026012420654, 0.719801127910614, 1.087682843208313, 1.3562037944793701, 0.05224694311618805, -0.8587559461593628, -1.2091511487960815, 1.0794813632965088, -0.24310223758220673, 0.9606706500053406, -0.544518768787384, 0.3932836651802063, 0.2269696444272995, 1.834641695022583, 0.8846904039382935, 0.2618323266506195, -0.9794660210609436, 1.2817381620407104, 0.9959952235221863, 0.06242821365594864, -0.2691572904586792, 2.3377997875213623, 0.5275061130523682, 0.49242210388183594, -0.8541580438613892, -0.5511642694473267, 0.8856618404388428, 0.2413529008626938, -0.4707653820514679, -0.13871021568775177, 0.496275395154953, 0.6307960152626038, 1.2531530857086182, -0.12244980037212372, -0.1853657215833664, -0.3159427344799042, 0.6607607007026672, -1.2647231817245483, -0.22306004166603088, 0.5400859117507935, 1.2307757139205933, 1.1479907035827637, -0.43654000759124756, -1.0607459545135498, 0.0033943711314350367, 3.2349531650543213, -0.7502182126045227, -0.07871739566326141, -0.056888893246650696, 0.7904114723205566, 0.02998984232544899, -0.28297311067581177, 0.3230627775192261, -1.5409940481185913, -1.481365442276001, 0.6561041474342346, -1.577001929283142, -0.34841254353523254, 0.020456500351428986, -1.0486546754837036, 0.624597430229187, -0.7077718377113342, 0.7478668093681335, 1.6761702299118042, -0.5601916909217834, 0.25970521569252014, -1.0871533155441284, -0.38181379437446594, -1.661155104637146, -0.34786713123321533, 1.5393856763839722, 1.4878567457199097, 0.6464974284172058, 0.005384839605540037, 0.09540614485740662, 0.9167593717575073, -0.5898927450180054, 1.3741906881332397, 0.6631590723991394, -0.2843480706214905, -0.08962468802928925, -0.2894892692565918, -1.693005919456482, -0.4547172486782074, 0.4575355052947998, -0.4986754059791565, 0.49048227071762085, 0.5693652629852295, -1.2388452291488647, 0.6097370982170105, 0.9376378059387207, -0.02559429034590721, -1.8557887077331543, 1.975541353225708, 0.14094915986061096, -0.814410388469696, -0.25399404764175415, 0.4726113975048065, 0.7177018523216248, 0.15397286415100098, 0.775835394859314, -0.10562162846326828, 1.078331708908081, -2.2917826175689697, -0.37807661294937134, -0.19537247717380524, 1.5533475875854492, -0.2942548990249634, 0.13818106055259705, -0.14816470444202423, 1.3073921203613281, -0.6262034177780151, 0.12588469684123993, -1.021103024482727, 0.21248140931129456, -0.20855006575584412, 0.16704778373241425, 1.8834275007247925, 0.5248440504074097, -0.3430820405483246, 0.528788685798645, 0.499434232711792], [0.24346251785755157, 1.8158100843429565, -1.3908673524856567, 0.22350899875164032, 0.5748525261878967, 0.6406745910644531, 0.055121053010225296, 1.0676987171173096, -0.7535619139671326, 0.8342933058738708, -0.5997576713562012, -0.24656131863594055, 1.8420151472091675, 0.0743146687746048, -0.2734808325767517, 0.5764958262443542, -1.1309982538223267, -0.2375669926404953, 0.3835296630859375, 0.9373559355735779, -0.6366310119628906, 0.611027717590332, 0.48543789982795715, 0.7454800009727478, 1.2346853017807007, -0.042296040803194046, 1.139784574508667, 0.1803755909204483, -1.6602720022201538, -2.469517230987549, 0.32806000113487244, -1.6923315525054932, -0.42171037197113037, 1.448424220085144, 1.098486065864563, -0.1355510950088501, 1.330586314201355, 0.5836775898933411, -0.7669290900230408, -0.16489087045192719, -0.820180356502533, -0.43266749382019043, 0.8389623165130615, 1.2407439947128296, -0.14334388077259064, -0.11946257203817368, 1.046420931816101, 1.7484967708587646, 1.5424509048461914, -2.8030552864074707, -0.8741109371185303, 0.06465453654527664, -0.34730005264282227, -0.08801931887865067, -0.13761202991008759, 1.130001187324524, 0.094573475420475, -0.37017399072647095, -0.05265083536505699, -0.4696340560913086, -1.161732792854309, 1.4925798177719116, -0.8264058232307434, 0.20273812115192413, 1.527226209640503, 1.9204117059707642, -0.008457913994789124, -0.43316352367401123, 0.5040779709815979, -1.3817620277404785, 0.46994253993034363, 1.1929693222045898, -0.5424303412437439, 0.9414427280426025, -0.0667598694562912, 0.3921078145503998, -2.456181764602661, 0.18328124284744263, -0.8368008732795715, 0.7234664559364319, -0.17127148807048798, -1.058729648590088, 0.7314148545265198, 0.40884125232696533, 0.5675426125526428, -0.678570032119751, -0.9069713354110718, 0.6741425395011902, -0.5919573903083801, -0.6566065549850464, -2.8065030574798584, 0.17210333049297333, -2.1075382232666016, -0.36523231863975525, 0.8404396772384644, 0.7664076685905457, -1.493019700050354, -0.8282594084739685, -0.42658138275146484, -1.579418420791626, 0.32608428597450256, 1.3543858528137207, 0.039947107434272766, -0.037109389901161194, -0.05196405574679375, -1.8932702541351318, 0.5971663594245911, 0.2887668013572693, 0.5153446197509766, -1.3010830879211426, 2.1357688903808594, 0.30052146315574646, -0.22987321019172668, -1.0983470678329468, -0.1399017870426178, 0.2746323347091675, -0.323621541261673, 1.0400232076644897, -0.2816029489040375, 1.0347108840942383, 0.8911577463150024, 0.5474333763122559, 0.8244339227676392, 1.6122876405715942, 1.3592275381088257, -0.20092499256134033, -0.1997714638710022, -0.6210092902183533, -0.991249144077301, 1.1199458837509155, 1.0683965682983398, 0.5239309072494507, -0.216266468167305, -0.30216315388679504, 1.2785805463790894, 0.31042560935020447, -0.26025906205177307, -1.135406732559204, 1.0057421922683716, 0.6536769270896912, 2.0617384910583496, -1.0151623487472534, -0.971636950969696, -1.5203505754470825, 0.28456515073776245, -0.23579974472522736, 0.8232702612876892, -2.666583776473999, -0.5737292766571045, -2.041748046875, -0.15093417465686798, -0.5040386319160461, 1.1794742345809937, -1.6720043420791626, -0.2980678379535675, -0.510880172252655, -1.5552939176559448, -0.31932127475738525, -0.5003088116645813, 0.7700657248497009, 1.0475656986236572, 0.44021058082580566, -1.0296406745910645, 1.5939182043075562, -0.4060341715812683, 0.6722273230552673, 1.7785030603408813, 0.4657912850379944, 1.538577675819397, -2.4167439937591553, 0.03909371793270111, 1.8706978559494019, 0.48781052231788635, 0.6910765171051025, -1.763780117034912, -1.632111668586731, 1.8186314105987549, 0.8372980952262878, -0.05655042082071304, 0.3940145671367645, -0.4377681314945221, 0.2885662019252777, 0.9159642457962036, 0.4565311670303345, 0.8659406304359436, -1.917496681213379, 0.6364832520484924, -2.0463037490844727, 0.8844783306121826, -0.8126576542854309, -0.5854882597923279, -0.3961072564125061, -0.7793146371841431, 1.2971506118774414, 0.959638774394989, 0.20410989224910736, -1.0941832065582275, 0.9977512955665588, 1.333864688873291, 1.6837364435195923, -0.438077449798584, -0.1924958974123001, 0.9743880033493042, 0.08724524080753326, 0.6040582656860352, 1.7217659950256348, -0.5728808045387268, -1.1489630937576294, -1.2110111713409424, -0.19023261964321136, -0.9081071615219116, 0.8205718994140625, -1.3462733030319214, 0.32756632566452026, 0.8973507881164551, 0.6112976670265198, 0.08987259119749069, 1.7746000289916992, 1.3471722602844238, -1.326896071434021, -0.16578760743141174, 1.5072118043899536, 0.15168096125125885, -0.407950222492218, 0.014623924158513546, -1.1037505865097046, -0.12203432619571686, -1.2717207670211792, 1.2909133434295654, -2.170736789703369, -0.6179250478744507, -0.9416411519050598, -1.8969106674194336, 0.37838125228881836, 0.7342574000358582, -0.14297714829444885, 1.0772135257720947, 0.1069868803024292, 0.34436559677124023, 0.6813403367996216, -0.06241234019398689, 0.6113559007644653, -0.13644905388355255, -0.74152010679245, -0.8293470740318298, -0.6507543921470642, 1.353015422821045, -1.5414083003997803, -0.6259250044822693, 0.06115354597568512, 1.4753347635269165, 0.4519558846950531, -0.37458330392837524, 0.346935510635376, 0.984830915927887, -1.1101092100143433, 0.3619173467159271, 0.3607269525527954, -1.1877751350402832, -0.5372834801673889, 0.0827425867319107, -0.851921558380127, 1.5322078466415405, -3.603672504425049, -0.4330322742462158, 1.1742357015609741, -0.05716085433959961, -0.5524419546127319, 0.4116586446762085, -0.02181134931743145, -0.31492921710014343, 0.6752645969390869, -0.5000126361846924, -1.1911617517471313, -2.2235171794891357, -0.16983014345169067, -0.4512542188167572, -0.35099104046821594, -0.2942636013031006, -1.4648281335830688, 1.218518614768982, 1.0332200527191162, 1.0094809532165527, 1.7818437814712524, -0.6899101138114929, 1.2784159183502197, 0.708092212677002, -0.6313738822937012, 0.8152850866317749, -1.331069827079773, -0.7227519154548645, 0.40652942657470703, -0.031102390959858894, 0.17119544744491577, -0.0877465233206749, -0.25727957487106323, -0.6605860590934753, -1.3890026807785034, 1.030447006225586, -1.1254770755767822, 0.8149438500404358, -0.41235777735710144, -0.8114379048347473, 0.17045234143733978, 1.4096049070358276, -0.4425647258758545, -0.661805272102356, -0.2433094084262848, 0.2019091099500656, -0.06597176194190979, 1.0230978727340698, 0.29522451758384705, -1.9660232067108154, -0.6727626323699951, -0.8479858040809631, 1.851348638534546, 3.2695977687835693, 0.18676799535751343, -0.11698684096336365, -0.07558651268482208, 1.0188640356063843, -1.204478144645691, 1.2999812364578247, -0.23738357424736023, -0.6107833385467529, 0.12928153574466705, -0.7854642271995544, -0.2813388705253601, -0.33064237236976624, -0.5978806018829346, 1.0668928623199463, 0.47846272587776184, 1.0750874280929565, -0.1066368967294693, -0.17182223498821259, -0.0031530018895864487, -0.3163829445838928, -1.0448797941207886, -1.7290321588516235, 0.5612380504608154, -0.35561108589172363, -0.3360721468925476, 1.153246283531189, 0.05618233606219292, -0.6813346743583679, 0.2850808799266815, -0.8253900408744812, 1.6275222301483154, -2.1450653076171875, 0.06895887851715088, 0.5345634818077087, -2.7540414333343506, 0.038237784057855606, 1.775085210800171, -0.7539610266685486, -0.6437832713127136, -0.1386362612247467, -0.09225950390100479, -0.43587538599967957, 0.21628855168819427, -0.0955498144030571, -1.3409616947174072, 0.0007376032299362123, -0.25791648030281067, 0.3010736405849457, -0.6462393403053284, -1.3149750232696533, 0.17747120559215546, 1.0146467685699463, -1.6608726978302002, 1.2513984441757202, -0.17140944302082062, -1.2006652355194092, 1.0965756177902222, 1.0558878183364868, -0.4873214364051819, -1.5551060438156128, 1.0659067630767822, 0.5221768021583557, 0.3193734288215637, -0.7500231862068176, -0.833483099937439, 1.246872901916504, 0.9977612495422363, 1.3508224487304688, 1.8425155878067017, 0.7092214822769165, 0.4655130207538605, 0.3087456226348877, -2.3067843914031982, -0.040205955505371094, -0.13676339387893677, 0.8350045084953308, 1.2021312713623047, 1.4750268459320068, 0.30307239294052124, 1.1691921949386597, -0.218673437833786, 1.72837233543396, 1.9077930450439453, 1.924595594406128, 1.4055899381637573, -0.7783717513084412, -0.5666160583496094, 0.7062156796455383, 2.9734702110290527, -0.497466504573822, 1.5158687829971313, 0.694989800453186, 1.136380910873413, 0.6465529203414917, -1.9041447639465332, -0.9572946429252625, 0.39957791566848755, 0.20064502954483032, 0.09136151522397995, 0.3953002393245697, 0.7381067872047424, 0.44449540972709656, 1.977220892906189, -0.3574977219104767, 0.1093253642320633, -0.16611650586128235, 0.4631161391735077, -0.3976747989654541, -0.268262654542923, -0.17956924438476562, 0.359634667634964, 0.17256125807762146, 1.408681035041809, -0.8324415683746338, 0.42877230048179626, -0.679608166217804, -1.2111400365829468, -0.2906107008457184, 1.527065396308899, 1.6681814193725586, 1.1413962841033936, -1.0377274751663208, 0.7599346041679382, -0.1317465603351593, -0.8899059295654297, 0.0685027614235878, -0.16466909646987915, 0.24743711948394775, 1.7117547988891602, 0.22035139799118042, 1.070719599723816, -1.7346670627593994, 0.7364541292190552, -1.670168399810791, 0.6788169741630554, -0.48666685819625854, -0.30537617206573486, 2.133875608444214, 0.5201604962348938, 0.5534443259239197, 0.7167907357215881, 0.6679620146751404, -0.5332850813865662, 0.4624771773815155, -0.4421834647655487, -0.1593896895647049, 0.23580946028232574, -0.3528142273426056, 1.1615797281265259, -0.9030320048332214, 1.164754867553711, -1.5569287538528442, -0.8338670134544373, 1.0563842058181763, 1.0723721981048584, 0.532075822353363, -1.2278755903244019, 0.712918758392334, 0.682999849319458, 0.7016329169273376, -2.113098621368408, 0.6390464901924133, 1.3551501035690308, -0.5420180559158325, 0.6812373995780945, -0.2876773178577423, -2.1933019161224365, -1.4167150259017944, 0.974970281124115, 1.1213595867156982, 0.3275318145751953, -1.3627148866653442, 1.380228042602539, -1.1906483173370361, -1.6234544515609741, 1.4090896844863892, -0.8565430045127869, 0.3315679132938385, 0.03213856741786003, 0.5977984666824341, 0.4550221562385559, -0.1919563114643097, -1.1365922689437866, -1.2444225549697876, -0.019403662532567978, -1.1492195129394531, -0.6655242443084717, -0.8647721409797668, -1.1404415369033813, -0.8560270071029663, 0.2034725546836853, -0.5793005228042603, -0.5157892107963562, 0.4714280962944031, 0.41368892788887024, -0.15795063972473145, -1.175654411315918, 0.8298823833465576, -1.6401174068450928, 0.29114648699760437, 0.6616243720054626, -1.7670010328292847, -1.3895649909973145, -0.8589774966239929, 0.4148060083389282, 1.253984808921814, -0.3699173331260681, -0.22446051239967346, -0.09475786983966827, 0.24569225311279297, -0.1424844115972519, 0.18935705721378326, -1.5453972816467285, -1.6714019775390625, -0.7640809416770935, -1.40081787109375, 0.5465376377105713, 1.0981775522232056, -0.13269107043743134, 1.0698270797729492, -1.4085010290145874, 1.1563280820846558, 1.2741252183914185, -0.05390682816505432, -0.6615304946899414, 1.8372255563735962, -1.0353784561157227, 0.24471043050289154, -0.4794284403324127, 0.7305654883384705, 0.8411983847618103, -0.11030300706624985, 0.9304426312446594, 0.4089552164077759, 0.6921485662460327, -2.0048718452453613, -0.9046328663825989, -0.9892588257789612, -0.5021160840988159, 1.5029536485671997, -1.0021761655807495, 0.43242937326431274, 0.8122861981391907, -0.6454948782920837, 0.5197544693946838, 1.185437560081482, 0.4859185218811035, -0.337695449590683, -1.0235422849655151, 0.7005805969238281, -1.4672131538391113, 1.5954132080078125, 0.1999833881855011, 1.3174928426742554, -1.3329178094863892, 0.2594354450702667, 0.9676883220672607, 0.8774104714393616, -0.7456764578819275, 1.396021842956543, -0.30736157298088074, 0.23322786390781403, 1.795828938484192, -0.39125344157218933, -1.3117393255233765, -0.9265238046646118, 1.895532488822937, 0.7746334671974182, -0.6430395245552063, 0.2907086908817291, 0.1657606065273285, 1.3052096366882324, 1.538773536682129, 1.0003787279129028, 0.12160183489322662, -1.6829535961151123, 0.3496708273887634, 1.098059058189392, 0.9277557730674744, 1.499376654624939, -0.34214362502098083, -1.4931267499923706, -0.8701789379119873, -1.1865397691726685, 0.19846348464488983, -1.4884835481643677, 0.4016937017440796, 0.010627501644194126, 0.729641854763031, -0.26414546370506287, -1.3014392852783203, -0.7339016199111938, -0.9611509442329407, 1.4207124710083008, 0.5269843935966492, -1.6261980533599854, -0.2564689815044403, -0.08342813700437546, -0.06105932593345642, -1.372255563735962, -1.2165831327438354, -0.1251516044139862, 0.22275564074516296, 0.019735217094421387, 0.8019751310348511, 0.4021763205528259, -0.7044446468353271, -2.170701265335083, -0.8769856095314026, -0.8698477149009705, -0.45375412702560425, -0.26319602131843567, -0.09853120148181915, -2.219830274581909, 1.182677984237671, -0.1981344074010849, 0.21215926110744476, -0.12384963780641556, -1.5737582445144653, -1.8371856212615967, -1.1161376237869263, 1.101362705230713, 0.9269155859947205, -1.195473074913025, -0.35850995779037476, -0.9643985033035278, 0.25558993220329285, -0.03358221799135208, 1.604074478149414, 0.6740095615386963, 1.8003642559051514, -0.7906448245048523, 0.8017372488975525, -1.6698721647262573, 0.8133293390274048, 1.0919705629348755, 1.7997255325317383, 1.2134888172149658, -1.6339261531829834, -0.36581552028656006, 0.34130898118019104, -0.9358441829681396, 0.5581143498420715, 0.9168446660041809, -0.22510188817977905, -1.1624610424041748, 1.0361618995666504, 0.9071488380432129, 0.09072928875684738, 0.006492303218692541, 1.9345710277557373, 0.8484811782836914, 0.19931206107139587, -1.3022124767303467, -0.9328525066375732, 0.6071224212646484, -0.13248547911643982, -0.43911266326904297, 1.466040015220642, 1.6413508653640747, -0.8120313882827759, 0.010431093163788319, 0.2939419746398926, -0.46660315990448, -1.9970935583114624, -0.8363160490989685, -0.2655441462993622, -1.3850706815719604, -1.085715413093567, -0.9347634315490723, -0.9382079243659973, 1.1445659399032593, -0.8360995054244995, -0.5440577864646912, 1.809574007987976, -0.4501511752605438, -0.352554053068161, -0.23456522822380066, 0.09300126135349274, 2.101588249206543, 1.7491697072982788, -1.1898618936538696, -0.0729655921459198, 0.0889185294508934, 0.4835730791091919, -0.6233798265457153, -1.011330485343933, -0.4391801655292511, 0.5446212291717529, -0.4146484136581421, -0.9085975289344788, -0.814053475856781, 1.1962963342666626, 0.14888955652713776, 1.982824444770813, 0.07570545375347137, 1.2930793762207031, 0.8587840795516968, 0.5256417393684387, 1.292331337928772, 1.1257225275039673, 0.2562044858932495, -0.05273453891277313, -0.6383540630340576, -1.4832578897476196, -1.8207480907440186, -0.9221320152282715, -0.832010805606842, -0.7620822191238403, 0.571861982345581, 0.18486231565475464, -0.785965621471405, 0.7696821689605713, -0.291630357503891, 0.11443464457988739, 2.0359723567962646, -0.4187432825565338, 0.9060670137405396, -0.43339604139328003, -0.5195761322975159, 1.065403938293457, -0.023879583925008774, -0.7094075083732605, -0.08613104373216629, -0.789216935634613, 0.3355618417263031, 0.280224084854126, -0.3682924211025238, 0.26106542348861694, 1.0781562328338623, -0.32626184821128845, -0.05500271916389465, -0.6519321799278259, 0.36517828702926636, -0.7934157848358154, 0.7435552477836609, 1.048438310623169, 0.0736997127532959, 0.8052098155021667, 0.17438730597496033, -0.911898136138916, 0.4176033139228821, -1.1929283142089844, 0.5475944876670837, 0.22273039817810059, 0.15216489136219025, 1.1895976066589355, -0.48658686876296997, -0.38036495447158813, -0.1500243991613388, 0.7836330533027649, -0.5075904130935669, 1.187146544456482, -0.332535982131958, 0.1540547013282776, -0.9819148778915405, 0.5080270171165466, -0.35611534118652344, -0.6196450591087341, 1.600758671760559, -2.26364803314209, -0.5966524481773376, 0.8234256505966187, 0.9067029356956482, 0.621757984161377, -0.1079377830028534, 0.044488757848739624, 0.12730492651462555, -2.6095352172851562, -1.247037410736084, -0.1315767914056778, -0.2548718750476837, -0.8242994546890259, 0.7807292342185974, -1.8433277606964111, -0.9714680314064026, -0.3403482437133789, 1.4632465839385986, -1.0874354839324951, -0.7961924076080322, -0.5012219548225403, -1.518471121788025, -2.0756258964538574, -1.3847342729568481, 1.0485767126083374, -1.0091886520385742, 0.05111750587821007, 1.3521759510040283, -1.4615269899368286, -2.0451412200927734, 1.6272482872009277, -1.8294429779052734, -2.0946125984191895, -0.06602772325277328, -1.3170924186706543, -0.2979520857334137, -0.32644206285476685, -1.308411717414856, 1.6355229616165161, 0.15176813304424286, -0.06462489068508148, -0.22795838117599487, -0.4142390489578247, -0.25675931572914124, 1.2560105323791504, 1.2893999814987183, -0.5813848376274109, -1.0531336069107056, 2.2097620964050293, 1.8017648458480835, 1.492620825767517, -0.7339223027229309, 0.12465056777000427, -0.255754292011261, 0.6192293167114258, -0.4042598307132721, 1.6021381616592407, 2.1061012744903564, 0.8111557364463806, 1.4655036926269531, 0.533326268196106, -0.24618728458881378, -0.842402458190918, -1.1574453115463257, -0.3410918712615967, 1.0407979488372803, 1.2731434106826782, -0.25738897919654846, 0.43452346324920654, 0.33020931482315063, -0.5594083070755005, 1.2141627073287964, -0.2805023789405823, -0.5296710729598999, 0.9346217513084412, -0.7737945318222046, 0.11986774206161499, 0.9936357140541077, -0.9065788984298706, -0.42281442880630493, 0.5953624248504639, -1.008975625038147, -0.5643147230148315, 1.803809404373169, -0.08182107657194138, -0.7976884841918945, -2.4850149154663086, 1.8504972457885742, -1.3913196325302124, 1.2642611265182495, 0.41160112619400024, 3.5465316772460938, -0.3680224120616913, 0.2713088393211365, 1.276617169380188, 0.14007346332073212, 0.9469262957572937, -2.3471267223358154, -1.1575473546981812, -0.1510477066040039, -1.0751513242721558, 0.5503208637237549, -0.742935836315155, 0.787305474281311, 2.7927284240722656, 0.7546480894088745, -0.3469201922416687, -1.4621717929840088, 0.21522150933742523, -1.322115421295166, -0.8451290130615234, -0.9111822843551636, 0.8092976212501526, -0.09157920628786087, 0.1911555826663971, -1.6580244302749634, 0.24461236596107483, 0.29214268922805786, -0.03160981833934784, 0.43463751673698425, -1.9535858631134033, -2.9860339164733887, 0.6267140507698059, 1.7515437602996826, 0.2033286988735199, -0.10013607889413834, 0.23006826639175415, -0.6873728632926941, -0.4032217264175415, -0.35673990845680237, -1.6672723293304443, -0.3321794271469116, 0.04805147275328636, 1.4757808446884155, -0.1904952973127365, -0.5788915753364563, 1.0898621082305908, -1.6673789024353027, 0.8494173288345337, -1.697467565536499, 0.12913675606250763, -0.6345148682594299, -1.06894850730896, -0.8831465840339661, 1.1701651811599731, 0.6393874883651733, -2.5578255653381348, -0.730725109577179, 1.7903680801391602, 1.659590244293213, -0.07742514461278915, -1.0440340042114258, -1.2266792058944702, -0.9681420922279358, 0.5468330979347229, 0.1424226015806198, -1.0507304668426514, -0.3616500794887543, -0.825702965259552, 0.48227617144584656, -0.05914808809757233, 0.5642728805541992, 1.8739573955535889, 0.3538390100002289, 1.5721728801727295, 1.450040578842163, -0.21584607660770416, -1.2346172332763672, 1.3566182851791382, -0.19871261715888977, -0.1827699840068817, -0.053479745984077454, -1.5776264667510986, 1.932979941368103, -1.073433756828308, 0.5783193707466125, -1.2471609115600586, 0.9364333748817444, 1.7016462087631226, -1.6310911178588867, -0.8119181990623474, 0.06913652271032333, 0.829708993434906, 0.6094627976417542, -0.04267321154475212, -0.4912761151790619, 0.8322927355766296, 0.9471147060394287, -0.14238165318965912, -1.1057920455932617, -1.7663811445236206, 0.9636358022689819, 1.2737815380096436, -0.3595567047595978, -0.718676745891571, 0.4866258502006531, 0.5087130665779114, -1.130734920501709, 0.4463141858577728, -0.5084456205368042, 1.3283685445785522, 0.3150404691696167, -1.3720967769622803, -1.7219932079315186, 0.6441618204116821, -1.2004873752593994, -0.654137134552002, -1.2209985256195068, 0.7513559460639954, 0.28485748171806335, 0.8611506223678589, -0.44051027297973633, 0.14151732623577118, 1.5892118215560913, -0.9823601245880127, 0.4179231822490692, 1.134218692779541, -0.23048171401023865, -0.29492297768592834, -0.9273940920829773, -0.10532448440790176, -2.589891195297241, 0.060879651457071304, -0.9638007283210754, 2.6727144718170166, -0.35646167397499084, 1.1413332223892212, -1.3541908264160156, -0.8947070837020874, 0.5140645503997803, -0.921413779258728, -0.7257867455482483, 0.7180888056755066, -0.25734391808509827, 0.828745424747467, -0.5821371674537659, -1.1298961639404297, -1.1233625411987305, 0.9204208850860596, 0.7884001731872559, -1.18514883518219, -2.798112392425537, 0.3435998260974884, 1.0338798761367798, 0.27760758996009827, 0.7656006813049316, -1.8509199619293213], [-0.18390785157680511, -0.6231682896614075, -1.6246119737625122, 0.0018572573317214847, 0.3674010634422302, -0.5363041162490845, -0.6857771873474121, 1.1568294763565063, -0.0030042980797588825, 1.6089539527893066, -1.7021652460098267, 0.5180898904800415, -0.6356091499328613, 0.9005599021911621, 0.5180733799934387, -1.7344353199005127, -1.0356385707855225, -0.6811137199401855, -0.22774899005889893, -0.15682676434516907, 0.8610383868217468, -0.43248656392097473, -1.3711732625961304, 1.1833397150039673, -0.5408505201339722, -1.1911259889602661, 0.1771237701177597, -1.1307295560836792, -1.267240285873413, -0.3683144152164459, 1.4368822574615479, 1.7745522260665894, 1.167128324508667, 1.0555976629257202, -0.15883871912956238, 1.2434635162353516, 0.68228679895401, -1.5737484693527222, 0.2785218358039856, -0.3312893211841583, 0.5552644729614258, -0.7803614735603333, 0.7339106202125549, -1.9132959842681885, 0.23753753304481506, -0.7229000926017761, 0.7825608849525452, 0.7131132483482361, 0.8401346802711487, 0.05343933403491974, -0.6551125645637512, 0.11578364670276642, -0.20819832384586334, -0.6763303279876709, -0.31587696075439453, 0.6243008375167847, -0.028331441804766655, -1.6047452688217163, -1.8791861534118652, -2.0890953540802, -0.9102540016174316, 0.8215051293373108, 0.3076954782009125, -1.5198042392730713, -0.21508927643299103, -1.930039644241333, 0.717526912689209, -0.19376911222934723, -0.022402051836252213, 1.9273853302001953, 0.22195753455162048, 1.3253942728042603, 0.4707569181919098, -0.14099079370498657, -0.37315070629119873, -0.09106817096471786, -0.48640912771224976, -0.7528948783874512, 1.293329119682312, -0.5947862267494202, -0.21879830956459045, -0.7491758465766907, 0.6126465201377869, -0.7242774963378906, 1.3256813287734985, 1.0485069751739502, -0.9056508541107178, -0.423539400100708, -0.10118252038955688, 1.4652202129364014, 0.9903276562690735, 0.30480489134788513, 1.946536898612976, 0.7975221872329712, 0.16527950763702393, 0.38407662510871887, 0.22028759121894836, -2.129594087600708, 0.3487992584705353, 1.3542017936706543, 0.04953723028302193, -1.3939154148101807, -0.03951750323176384, 0.12125303596258163, 0.1556013524532318, 1.8822407722473145, -0.6490492224693298, -0.907590925693512, 0.5243333578109741, 1.7955430746078491, -0.4424128532409668, 0.6823379397392273, -0.27945560216903687, -0.754574716091156, -1.389458417892456, -0.6678755879402161, -0.3217793107032776, 0.5668513774871826, 0.5196110606193542, -0.4139416217803955, 2.0721397399902344, 0.30133697390556335, -0.09790199249982834, -1.0546414852142334, 0.4138242304325104, -3.2964184284210205, -0.5824806690216064, 0.20721924304962158, -1.473590612411499, -2.027435541152954, 0.4279899001121521, -0.5042953491210938, -0.9611205458641052, -2.2133336067199707, 1.3375459909439087, -1.3904821872711182, -0.45386040210723877, 0.315555214881897, 0.25057253241539, 0.830888032913208, 0.1452367603778839, 0.8632016777992249, 0.8274624943733215, 0.7279365658760071, -1.5532851219177246, -0.5359622836112976, 0.23878620564937592, -1.1480587720870972, -1.2408744096755981, 0.429502010345459, -0.44474780559539795, 1.2946174144744873, 0.8929703831672668, 0.6189979910850525, 0.18158988654613495, -0.7514057755470276, 1.3757601976394653, 1.5052989721298218, 0.26975327730178833, -1.3531759977340698, 0.8523538112640381, -1.558252215385437, -0.17978335916996002, 0.38441869616508484, -0.4301997423171997, -0.01660393364727497, 0.376583993434906, -0.938985288143158, 0.3668912947177887, 0.8782010674476624, 3.0716166496276855, -0.7469897270202637, 1.0542539358139038, 0.5294176936149597, -0.14529873430728912, -0.7221772074699402, -0.3548991084098816, -1.6422332525253296, -0.18539726734161377, 0.6415025591850281, 0.285827100276947, 0.79217928647995, -0.6543945074081421, -0.4680232107639313, 0.1012539342045784, 0.351894736289978, 1.6031497716903687, -0.6811054348945618, 0.655414879322052, 0.8884294629096985, -0.3908052444458008, -0.4176218807697296, 1.0725387334823608, 1.7317140102386475, 0.38117390871047974, 1.077325463294983, -0.6045621633529663, -0.47874101996421814, 1.2434239387512207, -1.8923747539520264, -0.21677590906620026, -1.7584846019744873, -1.6339497566223145, 0.8666690587997437, 0.6813070178031921, 0.14242973923683167, 0.8004425168037415, 1.2022348642349243, -0.7719857096672058, 0.3196059763431549, -0.8675777316093445, -1.559087872505188, -0.16208454966545105, 0.7559071183204651, 0.8636296391487122, -0.4386803209781647, -0.029247606173157692, 0.005569301545619965, 0.29118993878364563, -0.3874622583389282, -0.18777580559253693, 1.450320839881897, 0.37004849314689636, -0.6952072381973267, 0.5558111071586609, 0.5166229009628296, 0.21788175404071808, -0.6356534361839294, -0.1882552057504654, -0.10182684659957886, 2.5470173358917236, -0.4899972975254059, -0.9479665756225586, -0.42027750611305237, -0.3484063148498535, 0.17562632262706757, -1.4095572233200073, 0.3119293451309204, 0.44917646050453186, -2.081751585006714, -0.047311462461948395, 0.6708376407623291, -0.15246091783046722, 0.26382139325141907, 1.4863510131835938, 2.0408244132995605, 0.020856432616710663, 0.04918524995446205, 0.22372953593730927, 1.1076991558074951, 0.1711702197790146, -0.5607624053955078, -1.338387131690979, -1.9945398569107056, 0.01725863106548786, -0.30141374468803406, 0.8682923316955566, 0.11863579601049423, 0.2568640410900116, -0.6015786528587341, -1.5974444150924683, 0.7999622225761414, 1.4703584909439087, -1.3154627084732056, -0.9830437302589417, 0.7849410176277161, -0.3909817337989807, -0.46728360652923584, 0.1652221828699112, -1.7398133277893066, 1.5209770202636719, -0.46612292528152466, -1.213807463645935, -0.4957907795906067, 0.8692249059677124, -0.3497641086578369, 0.6020666360855103, 1.4411662817001343, -0.5030575394630432, -1.3922337293624878, -0.6193727850914001, 1.5446720123291016, -1.0227150917053223, -1.1809040307998657, 0.7274278402328491, -0.5040797591209412, -1.1246248483657837, 0.021397555246949196, -0.4527686536312103, 1.2839500904083252, -0.7998228669166565, 0.5100439786911011, -0.11836165189743042, 1.3244937658309937, 0.39447376132011414, 0.5997625589370728, 1.337122917175293, -1.6497911214828491, -0.18244239687919617, -0.8988334536552429, 0.872566282749176, -1.750876545906067, 1.442908763885498, 0.40661704540252686, 2.974219560623169, 0.35761693120002747, 0.10288851708173752, -0.4657241702079773, -0.8957924842834473, 0.4675637483596802, -1.2206791639328003, 0.3639540374279022, -1.2767637968063354, -0.982304036617279, -0.12272032350301743, -0.5764845013618469, 1.1034692525863647, -0.50211501121521, -0.3248305916786194, 1.0970561504364014, -0.1074940636754036, 0.40542805194854736, 0.7863932251930237, 0.07174957543611526, -1.3878365755081177, -0.4981153607368469, -0.5841754078865051, 1.3817387819290161, -0.5578230023384094, 0.2190694957971573, -0.9650804400444031, -2.0956785678863525, -0.6454962491989136, -0.6855522990226746, -1.9171901941299438, 0.19142980873584747, -0.1440313458442688, -0.9398687481880188, -1.0810571908950806, -1.5660501718521118, 1.0822819471359253, -0.5454153418540955, 0.3212992548942566, -0.3312326967716217, 1.1379643678665161, 1.4385950565338135, -1.2471963167190552, 2.710662841796875, -0.6240313053131104, -0.8603612184524536, -0.7761325240135193, -0.5485363006591797, -0.7605567574501038, -0.12199055403470993, -0.19652439653873444, 0.37956881523132324, -0.7180606126785278, -0.563069224357605, -0.2904216945171356, -0.6649587154388428, 0.784361720085144, -1.1940959692001343, -1.0810234546661377, 0.9683154225349426, -0.2117587774991989, 1.0184879302978516, -0.6617613434791565, -0.8598377704620361, -0.4546865224838257, -0.08030006289482117, -1.6127021312713623, 0.03579830750823021, -0.6351044774055481, 0.16604293882846832, -0.3760530948638916, 0.19300617277622223, 1.211216688156128, -0.5739207863807678, -0.12128706276416779, 0.6630641222000122, -0.6175454258918762, -0.9226073622703552, -0.5824604630470276, -0.21062374114990234, -0.5044827461242676, -1.4913344383239746, 3.064383029937744, -0.17658165097236633, 1.8067344427108765, -1.1788331270217896, -0.6794614195823669, -0.008430120535194874, -0.2731572687625885, -2.208468198776245, 0.16126900911331177, 1.3221131563186646, 1.3532665967941284, -0.34531599283218384, 1.3529918193817139, -1.294817566871643, -0.42815637588500977, -0.4224560558795929, 0.4394586384296417, -0.0707419216632843, 0.3928717374801636, 0.3631172180175781, -0.5635985732078552, 1.1796189546585083, -1.5353845357894897, 0.8587061166763306, 0.8498138785362244, -0.3260958194732666, 0.6862543821334839, -1.320713758468628, -2.063868999481201, -0.1778344064950943, 1.1823638677597046, -0.27518486976623535, 0.5177003741264343, -0.34194332361221313, 1.002766489982605, -0.6833995580673218, -0.18548640608787537, 0.9199202656745911, 0.5666633248329163, 1.3863173723220825, 1.948356032371521, -0.044353894889354706, 0.7841282486915588, -0.021820498630404472, 0.8143517374992371, 0.15500663220882416, 1.5986762046813965, -0.24673675000667572, -1.7850546836853027, -1.3889483213424683, 1.2660702466964722, 0.5557945966720581, -0.22216427326202393, 1.3256341218948364, -1.9286291599273682, -0.08526600152254105, 0.5866346955299377, 1.4410468339920044, 0.48627084493637085, -1.0739229917526245, -0.008334335871040821, -0.7796031832695007, 3.3273417949676514, 0.9036588072776794, -1.4311108589172363, 1.1329457759857178, 0.4601418077945709, 0.45612192153930664, 0.42698749899864197, 0.5230082273483276, -1.279934287071228, -0.8544880151748657, 1.3793565034866333, -1.0918463468551636, 0.9356834888458252, -0.5912068486213684, 1.2070261240005493, -0.02258886583149433, 1.0174256563186646, 2.6962790489196777, 0.31159985065460205, 0.8261502981185913, -0.1854749619960785, -0.9357757568359375, -0.405015230178833, -0.46889394521713257, 0.16633766889572144, 0.6771889925003052, -2.588322162628174, 0.5023858547210693, -0.34539252519607544, 0.9899343252182007, -1.5666276216506958, 1.479946494102478, 0.9884504675865173, 0.10738716274499893, -1.2883660793304443, -0.36024051904678345, -0.6763144135475159, 0.4880492389202118, 0.5818268656730652, 0.9144208431243896, -0.060209620743989944, -0.2618432939052582, -2.4826292991638184, -0.008908988907933235, 0.02738035097718239, 0.5510228872299194, -0.16764579713344574, 0.40440404415130615, 0.40507322549819946, 0.8027912378311157, -0.9797425866127014, 1.480466365814209, -1.3348190784454346, 1.6765447854995728, 0.4267420768737793, -0.16140039265155792, -0.9012235403060913, -1.1821913719177246, -0.888900876045227, -0.03788992390036583, 1.3532618284225464, -0.13027891516685486, 0.29893720149993896, 0.1008772999048233, -1.4497650861740112, -1.4972082376480103, -1.2103910446166992, 0.2020149827003479, 0.9467082619667053, -0.30725762248039246, 0.42040011286735535, -0.6866721510887146, -0.2598531246185303, 1.2488608360290527, 2.5178186893463135, 0.6076313853263855, 0.08718845248222351, -0.1622541844844818, -1.721972107887268, -1.529954433441162, 0.42954379320144653, 0.784476101398468, -1.3291014432907104, 0.05698814243078232, 0.15459968149662018, -1.1761568784713745, -0.1694737821817398, 0.008299079723656178, 0.9449131488800049, 0.7749806642532349, -0.04393518343567848, -0.809794008731842, 0.3516324460506439, -0.25925159454345703, -0.8945410847663879, 0.6503493785858154, -1.2386009693145752, 0.7886742949485779, 1.036617398262024, 0.19040636718273163, 0.059720348566770554, -0.5484844446182251, 0.2610420286655426, -1.88205087184906, -0.8299043774604797, -1.4354572296142578, -0.1972123235464096, 0.12380865961313248, -0.7032204270362854, -1.1207449436187744, 2.125502347946167, -0.7424986958503723, -0.9415866732597351, -0.15987926721572876, 1.0227172374725342, -0.8465414047241211, 0.9645240902900696, 1.3204680681228638, -1.1352946758270264, 0.624161958694458, -1.2089734077453613, 1.8744394779205322, -0.47171199321746826, 0.6096482872962952, -2.182708740234375, -0.9497702121734619, 2.191180944442749, 0.07939466834068298, 0.16487286984920502, 0.5617919564247131, 0.01994974911212921, 0.25791990756988525, -0.44034814834594727, -1.0053642988204956, -2.88612961769104, 1.7560030221939087, 2.7302374839782715, 0.34438660740852356, 0.8902965784072876, 0.9399765729904175, 1.5690760612487793, 0.9967538714408875, 0.9103809595108032, -0.5998508930206299, -0.17105737328529358, 0.7959423661231995, -0.15566642582416534, 0.06488039344549179, -0.1549399346113205, -1.0819203853607178, 1.2523078918457031, 0.4604533910751343, -0.3558707535266876, 1.9540767669677734, -0.5746517777442932, -0.3829706609249115, 0.3446762263774872, 0.4497624635696411, -1.085982322692871, -1.196303129196167, -1.3887231349945068, 0.20017214119434357, 0.08053788542747498, 0.9108067154884338, -2.0469090938568115, -0.29839232563972473, -0.9281283617019653, -0.3080956041812897, -0.2399146854877472, 0.2287561297416687, 0.7380764484405518, -1.7469402551651, -1.4988179206848145, -0.8279169201850891, -1.2839685678482056, 0.5092958807945251, -0.1341070830821991, -2.9934027194976807, -0.10293781757354736, -0.5446191430091858, -0.0031738360412418842, 0.2397363781929016, -1.6802855730056763, 2.3644094467163086, -0.26576852798461914, -0.3956136703491211, -0.6950848698616028, 0.8307872414588928, 0.44611856341362, -0.9491406679153442, 2.100410223007202, 0.4659000635147095, 0.7709241509437561, -0.846623957157135, -0.8665693402290344, 0.03677332028746605, 1.4760581254959106, -2.0623111724853516, 0.08078990131616592, -1.1141220331192017, 1.0675747394561768, -0.1487739384174347, -0.3334695100784302, -0.07997728884220123, -0.2830486595630646, 0.27023443579673767, 0.057476118206977844, 0.18953996896743774, -0.8316280841827393, -0.507940948009491, -0.4243435859680176, 1.9405262470245361, -1.6762114763259888, -1.461856722831726, -0.7322906255722046, -0.1775357574224472, 0.6576744914054871, 1.0348544120788574, 0.34315600991249084, -0.015151966363191605, -1.5663776397705078, 0.7902633547782898, 0.2951403856277466, -1.0830321311950684, 1.643507719039917, -1.046302080154419, -1.3828312158584595, 0.8212178945541382, -1.3127487897872925, 0.32133522629737854, -0.367267370223999, 0.8811774253845215, -2.022894859313965, -1.0324264764785767, 0.0070887100882828236, 0.4094771146774292, 0.52129727602005, -1.3072283267974854, -0.6016317009925842, -0.9912216067314148, 0.1863624006509781, 2.0085413455963135, -0.16711953282356262, -0.7806550860404968, 0.3878491520881653, 1.0088831186294556, 2.0170886516571045, -0.9623778462409973, 1.7180095911026, -1.0324972867965698, -1.0105628967285156, 0.7942258715629578, 0.6873015761375427, -0.9176350831985474, -0.7542906403541565, 0.5953274369239807, 0.6932212114334106, 0.033932559192180634, -0.865011990070343, 2.3565673828125, 0.0925748199224472, 1.060757040977478, 0.08392123878002167, 0.6051703691482544, 0.7266941666603088, -0.8583000302314758, 0.2530103623867035, 0.650039553642273, 0.015441790223121643, 0.545703113079071, 0.040892768651247025, -0.11508899182081223, -0.5362365245819092, 0.4139627516269684, 0.9755070209503174, -1.3787713050842285, 0.5230551362037659, -0.24664953351020813, 0.8956593871116638, 0.22891810536384583, -1.9073776006698608, -0.2057119905948639, -1.668508768081665, -0.39431336522102356, 0.7574205994606018, 1.0332494974136353, -0.8693211078643799, 0.4906625747680664, 0.5050472617149353, -0.45747068524360657, -1.358291745185852, 0.19687914848327637, 0.8353198766708374, -0.060533225536346436, 0.48955097794532776, 1.340995192527771, 0.013165893033146858, -0.8221022486686707, -2.3078720569610596, 0.4070460796356201, 0.4129910469055176, -0.6271745562553406, 0.07514216750860214, -0.6854872107505798, 0.03851836919784546, -1.5656299591064453, 0.4537881314754486, 0.5109237432479858, 0.36734920740127563, -1.7400496006011963, 0.48654475808143616, -2.5518991947174072, 0.037341777235269547, -0.3245886564254761, 0.4187832176685333, -0.4847128987312317, -0.5162097811698914, 0.18050330877304077, -0.7363616228103638, -0.11896242201328278, 1.5074162483215332, -0.6675826907157898, 0.3378216326236725, 0.22325962781906128, -0.8145713210105896, -0.08463069051504135, 0.33978721499443054, 0.30493491888046265, -1.5732115507125854, 1.8193460702896118, 0.044783852994441986, 1.2649502754211426, -0.41773688793182373, 0.21456533670425415, -1.2406816482543945, 0.7583404183387756, -1.1517192125320435, 0.6499471068382263, 1.004584550857544, 0.9774479269981384, -0.48048898577690125, 1.042409062385559, -0.5365712642669678, 0.8368505239486694, 0.445391982793808, 0.4512636661529541, 1.7120239734649658, -0.5158815383911133, -1.8119527101516724, -0.34632113575935364, -0.3859897553920746, -0.918502688407898, -0.42507821321487427, 0.15594954788684845, 1.225540280342102, 2.771320104598999, -0.2886207699775696, -0.17037713527679443, 0.9843646287918091, 0.9257363677024841, -0.5258125066757202, 1.1058199405670166, 0.8557974100112915, -0.4011928141117096, -2.6182031631469727, 0.3589552342891693, 0.21933475136756897, -1.1260528564453125, -1.583022952079773, 1.3345590829849243, -2.1586451530456543, -0.22894041240215302, 1.4463783502578735, 0.5680606365203857, -1.4019044637680054, 2.8746931552886963, -0.9714366793632507, -1.4172579050064087, 0.6527710556983948, 0.755182683467865, -1.6776442527770996, 0.5803145170211792, 0.2035522162914276, -0.1293349415063858, -0.7902050018310547, 0.45844557881355286, -1.8584753274917603, 0.788632869720459, 0.3047819137573242, -0.6214386224746704, -0.9534053206443787, -0.10889003425836563, 2.1185951232910156, 0.4202490746974945, 0.07544641941785812, -0.3199441432952881, 0.5073164105415344, 1.4934066534042358, 1.0747113227844238, -0.03881978988647461, -0.2807237207889557, 0.32905641198158264, 0.4391999840736389, -1.6413944959640503, 0.3831697702407837, 0.5833450555801392, 1.4392669200897217, 0.3625905215740204, 0.26710405945777893, -0.28800395131111145, 0.06968940049409866, -0.21602743864059448, 1.6544201374053955, 0.18522413074970245, 0.5064005851745605, -1.0615348815917969, 2.2770185470581055, 2.755460023880005, 0.7895168662071228, -1.1123151779174805, -0.5149060487747192, -0.5282920002937317, 0.2729688584804535, 0.3808400332927704, 0.7234913110733032, 0.3016415536403656, 0.5680879354476929, -1.3113529682159424, 2.0742554664611816, -0.004033226985484362, -0.06377630680799484, 1.219971776008606, 0.730376660823822, -0.9213436245918274, 1.1606650352478027, -2.100182294845581, 0.7545793056488037, -0.5190439820289612, -2.0055413246154785, 0.002308623166754842, -0.483278751373291, -1.8319693803787231, 0.6228275299072266, 0.016387581825256348, -0.6919618844985962, -0.4044454097747803, -0.4784427583217621, -0.6064848303794861, 0.5787943601608276, 0.5104135870933533, -0.26260918378829956, -0.9383999705314636, 0.17557010054588318, -2.043074369430542, 0.9302998781204224, -0.46446651220321655, -0.3482438027858734, -0.7658670544624329, -1.4488379955291748, 0.33970722556114197, 0.7936117649078369, 0.45370879769325256, 0.15595269203186035, -2.142946720123291, -1.3911070823669434, 0.32355013489723206, 0.15786170959472656, 0.2432374507188797, 0.06314940750598907, 0.2953636646270752, -0.8786131739616394, 1.574724555015564, 0.43403226137161255, -0.32860681414604187, -1.0024429559707642, -0.5327993631362915, -1.8567452430725098, -0.5854460000991821, -0.4141344428062439, -0.3116282820701599, -2.3640525341033936, -0.8896234631538391, 0.05816650390625, 1.5529721975326538, 1.3131508827209473, -2.3452494144439697, -0.697228193283081, -0.2713901698589325, -0.7703859210014343, -1.0918179750442505, 0.41993772983551025, 1.0426032543182373, 0.175539031624794, 0.4293873906135559, 0.5647360682487488, 1.1750988960266113, -0.8368695974349976, 1.3296979665756226, 1.39421546459198, 0.05294997617602348, 0.2314223051071167, 0.5534520745277405, -1.2629764080047607, -0.09459252655506134, 0.5913006067276001, -0.04151336848735809, 0.015013100579380989, 0.27703022956848145, 1.1921212673187256, 0.2765301764011383, -1.0442074537277222, 1.1033563613891602, 0.41400307416915894, 0.6742931008338928, -0.268498957157135, 0.7703145146369934, 0.5391089916229248, 0.2932310402393341, 1.637261152267456, -1.0678759813308716, 0.3296550512313843, -0.01987582817673683, 0.679968535900116, 0.23824873566627502, 0.6500673890113831, -0.8170393109321594, -0.15311987698078156, -0.9454761743545532, -1.042365312576294, -0.5088838338851929, 0.5016154050827026, -0.7064409255981445, 0.23562170565128326, 1.5461357831954956, 0.388992041349411, 0.15996813774108887, -1.501611351966858, -1.9127912521362305, 0.24212904274463654, -0.30198025703430176, 0.14205017685890198, 0.38261932134628296, 1.025547742843628, -0.7229099869728088, -0.4386889934539795, 1.3261343240737915, -0.11062029004096985, 1.6635791063308716, 1.2312750816345215, -0.8763678073883057, 0.09609207510948181, -0.11486982554197311, -0.15058165788650513, 0.9705681204795837, -0.4709349274635315, 0.5382378697395325, -0.2602716386318207, 0.04360635578632355, -0.4315965473651886, -0.19543780386447906, 1.405558705329895, 0.8100302219390869, 0.41499271988868713, -1.5705622434616089, 1.2118523120880127, 2.888563871383667, -0.6953265070915222, 1.7279551029205322, 1.1570359468460083, -0.1280607134103775, -0.5846706628799438, -0.8235878944396973, -1.6737416982650757, -0.20602938532829285, -0.4533214867115021, 0.5530375838279724], [-0.3871307075023651, 0.8775854110717773, -0.34090355038642883, -2.542710065841675, -0.029884861782193184, 0.04741792380809784, 0.24544088542461395, 1.4807040691375732, -0.9595791101455688, -0.477853000164032, 0.9227129220962524, 0.8615585565567017, 0.3483019769191742, -1.1675145626068115, -0.08669544011354446, -0.2528979182243347, -0.18812011182308197, -0.505566418170929, 0.29691264033317566, -0.38593873381614685, 0.19178324937820435, 0.7448924779891968, -1.3194403648376465, -0.019482901319861412, 0.03825700283050537, 0.575836718082428, -1.634874939918518, 0.87790846824646, -0.003710670629516244, -0.33017557859420776, 0.12375407665967941, -0.09550537168979645, -1.4100638628005981, -0.7045484781265259, 0.15936973690986633, 0.3893161714076996, -0.03579653427004814, 1.3874800205230713, -1.0916670560836792, 0.22541651129722595, 1.6433335542678833, -0.668319582939148, 0.3833359479904175, 0.40319710969924927, -0.9833505749702454, -0.13359515368938446, 0.8716267943382263, -0.10898984968662262, 0.6488546133041382, -1.1645179986953735, -0.0605253130197525, -0.727070152759552, 0.4547680914402008, -0.7792673707008362, 1.7866355180740356, -0.48544639348983765, 0.7520754337310791, -0.017644314095377922, -2.232635021209717, 0.1493515521287918, 0.13362430036067963, -0.04509827867150307, -1.9650366306304932, 0.0833275318145752, -1.1944433450698853, -0.8156276345252991, -0.2197178453207016, 0.3534960448741913, 1.083174467086792, -0.40812602639198303, -0.3864799737930298, 1.091688871383667, 0.5884227156639099, 0.19718442857265472, -0.5932243466377258, 0.19517366588115692, 0.2646807134151459, 0.34410443902015686, -0.1475907564163208, 1.3295233249664307, 1.460716724395752, 0.12729433178901672, 1.1088088750839233, 0.19182555377483368, -0.13985557854175568, -0.08079691231250763, -1.5542811155319214, 0.9096709489822388, -0.7343651056289673, -0.3083139657974243, -0.3973977565765381, -0.47460827231407166, -0.3145647943019867, -2.1275713443756104, -0.05282934010028839, -0.389595627784729, 0.8545666337013245, 1.2795504331588745, -1.269883632659912, -0.5020554661750793, 0.08949697762727737, 1.7537128925323486, 0.6687983870506287, -0.6966503262519836, 0.23487718403339386, -0.5415438413619995, -0.24919411540031433, 0.017760753631591797, -1.629210352897644, -0.8286293148994446, 0.012148411944508553, 0.6203259229660034, -0.3045283555984497, 0.5201606750488281, -0.2204066663980484, 0.10208554565906525, -0.9370266795158386, 0.9046756029129028, -0.7184092998504639, -1.638085961341858, 1.0959731340408325, 0.07175423204898834, -0.909416675567627, -0.3875027596950531, -0.18542785942554474, 0.43965670466423035, -0.38717448711395264, -0.5778427720069885, 1.4056272506713867, 0.23171739280223846, -0.9459546208381653, -0.25178566575050354, -0.39541178941726685, 0.42157426476478577, -0.8750623464584351, -1.0978566408157349, 1.1412957906723022, 0.9372876286506653, -0.4864545464515686, -0.44459789991378784, -0.7465115189552307, -1.1001261472702026, -1.2222890853881836, 0.07191445678472519, 1.1636385917663574, 0.01418954599648714, -1.26075279712677, 1.0000925064086914, 1.2988272905349731, 0.631005048751831, -1.467582106590271, 0.5676453113555908, 0.008346563205122948, 1.1576097011566162, -2.395662784576416, -1.3020541667938232, -1.0758486986160278, 1.9096558094024658, 0.038529135286808014, 0.3834831416606903, -0.8282343149185181, -0.05446234717965126, -0.27300333976745605, 1.1554759740829468, 0.3655150532722473, -1.3821725845336914, 0.6566718816757202, -0.477308452129364, 0.05316844955086708, -1.473564863204956, 0.6543201208114624, 0.7600650787353516, -0.931494951248169, 0.01616230048239231, 0.1988677978515625, -0.5311504602432251, 0.25048068165779114, -0.4185633063316345, -0.355609655380249, 1.2161576747894287, 1.7193979024887085, 1.1977602243423462, 1.320088505744934, -0.14790725708007812, -0.5507042407989502, 1.2955793142318726, 1.3827729225158691, -0.4037724435329437, 0.19635480642318726, 0.2256917804479599, 1.4581255912780762, -0.7248750925064087, -0.35841503739356995, -0.16824021935462952, -0.8512499928474426, 0.06275557726621628, 0.6811735033988953, 0.13044004142284393, 0.18523971736431122, 0.10466533899307251, -0.6468601226806641, -0.5316122770309448, 0.3527958393096924, -1.6283378601074219, -0.006451237481087446, -2.089519739151001, -1.0223051309585571, -1.0167828798294067, 0.9482669830322266, -0.02289257012307644, 1.0070093870162964, 1.132344126701355, 2.067301034927368, -0.33401742577552795, 0.6520851850509644, -0.9158729910850525, 0.45545071363449097, -1.1594880819320679, -1.1077061891555786, -1.1419187784194946, -0.2500745952129364, -0.17447786033153534, 1.0509445667266846, -0.9768192768096924, 0.5574509501457214, -0.6302253007888794, 1.445155143737793, 0.1637801229953766, 1.4365103244781494, 1.4625930786132812, -0.45194828510284424, 1.1007238626480103, 0.6075476408004761, 1.0360063314437866, 0.7018885612487793, -1.3610318899154663, 0.20529237389564514, 0.8744894862174988, -0.6201031804084778, 0.5167211890220642, -0.11992628872394562, -1.6138321161270142, -1.4114116430282593, 0.28977739810943604, 0.39982762932777405, 1.4192383289337158, -0.6375528573989868, -0.4664308428764343, 1.244952917098999, -0.21088556945323944, 0.5085240006446838, 1.1566975116729736, 0.9972941279411316, 0.14741024374961853, -0.7287566661834717, -1.3557138442993164, -1.2759616374969482, 0.8704968690872192, 0.08950251340866089, -0.05586633086204529, 0.5529441833496094, -1.248311161994934, -1.786907434463501, 0.2245454341173172, -1.3084567785263062, -0.5554769039154053, -0.3285650610923767, 1.2050971984863281, 0.39123228192329407, 1.8923470973968506, -0.9316221475601196, -1.068599820137024, -1.279720425605774, -2.2471745014190674, 0.318702757358551, -2.1294875144958496, 0.3237870931625366, 2.067678451538086, 1.158706784248352, 0.04314861819148064, -0.6949909329414368, -0.2128393054008484, 0.3214309811592102, -0.018508873879909515, -0.8953465819358826, 1.4310073852539062, -0.38952529430389404, 0.6582688689231873, 0.6790242791175842, -0.7792633771896362, 0.8562184572219849, -1.424583077430725, 1.4833904504776, -0.16269749402999878, -0.012406869791448116, -2.0713109970092773, 0.550871729850769, 1.4760794639587402, -0.13567839562892914, -1.254517912864685, -0.9371456503868103, -0.34333908557891846, -0.29345813393592834, -0.6307698488235474, 0.5412607192993164, -1.1627790927886963, -0.21999414265155792, -0.736720085144043, 0.9747363328933716, -0.1011601984500885, 1.6174235343933105, 0.04805723577737808, 1.4817460775375366, 0.9239906668663025, -1.2046619653701782, -0.40075451135635376, 1.2033538818359375, 0.9817447662353516, -1.6874442100524902, 0.009530741721391678, -0.9975926280021667, -1.5941638946533203, 0.3160156011581421, -0.322025328874588, 0.09258191287517548, 1.2927076816558838, 0.7467998266220093, 0.966827929019928, 1.0307742357254028, 0.11993009597063065, 1.0111838579177856, 0.9228739142417908, -0.606637716293335, 1.1671137809753418, 1.654877781867981, 0.2562759816646576, 0.03996458649635315, 0.3709243834018707, -0.28806272149086, -0.20634615421295166, -1.3014768362045288, -0.12688852846622467, 1.359560251235962, -1.2454109191894531, 0.5094168782234192, -1.037164568901062, 1.1960903406143188, -0.9924163818359375, -1.2425005435943604, 0.8823376297950745, -0.6286258101463318, -0.8836807608604431, 0.2257426530122757, 0.11978233605623245, 1.1893390417099, 0.11846037209033966, -0.9497983455657959, -0.2982341945171356, -0.424512654542923, -1.002357840538025, 0.4156976044178009, -0.5967214107513428, -0.31812265515327454, -0.19255541265010834, -2.218508243560791, 0.799178421497345, 3.0589473247528076, 0.32196125388145447, 0.9755753874778748, 1.1848523616790771, 0.28307482600212097, -2.3155767917633057, 1.21436607837677, 2.690570116043091, 1.2348271608352661, 0.20707125961780548, 1.1630374193191528, 0.9138636589050293, 0.6488252878189087, 1.8909645080566406, -0.9673368334770203, -0.6368910074234009, 0.015407651662826538, 1.5666265487670898, 0.1836530715227127, -0.9530201554298401, -1.061924934387207, 0.02466818131506443, 0.96440589427948, -0.2154911905527115, -1.56861412525177, -0.5462985038757324, 0.41433659195899963, -1.170562505722046, 0.8467053174972534, 0.7134132385253906, 1.3439112901687622, -0.943523108959198, 1.0071693658828735, 0.04555799812078476, -0.7690654397010803, -0.3784016966819763, 0.1709628701210022, -0.4260457158088684, 1.0958493947982788, 0.2600754201412201, 1.1168322563171387, 0.4881702959537506, -0.36880984902381897, -0.7781834602355957, 1.0983407497406006, 0.2958839535713196, 0.4149188697338104, 0.07274923473596573, -0.053888969123363495, -0.251050740480423, -1.7061991691589355, 0.1637607216835022, -1.0807143449783325, 0.3136860430240631, -0.46820420026779175, -1.4225795269012451, -1.2087583541870117, 0.7368108630180359, -0.17075589299201965, -0.5563949346542358, -0.7080830335617065, -0.8403289914131165, -0.49490123987197876, -0.4584529995918274, 0.5106710195541382, 0.4874141216278076, -1.020007610321045, -1.0823118686676025, -0.3728652894496918, 0.5686426758766174, -0.2658919394016266, 0.04196317866444588, 0.5384684801101685, -0.4163580536842346, -0.12191879749298096, -0.7552469372749329, 1.082068920135498, 0.9760292172431946, -0.6608229875564575, 1.8017598390579224, 0.10339374840259552, -0.8551176190376282, -0.28550878167152405, -0.08180318772792816, 0.7816800475120544, -1.2121542692184448, -1.4140641689300537, -1.710646390914917, -1.2444292306900024, 0.799662172794342, -0.5619944334030151, -0.37797293066978455, -0.3000897169113159, -0.042522870004177094, -0.9115609526634216, 1.406032681465149, -1.191151738166809, -1.6237772703170776, -0.8927679657936096, -1.636081576347351, 0.44870540499687195, 1.2756725549697876, 0.380641371011734, -1.3607996702194214, -1.0793242454528809, -0.37671059370040894, -0.6841900944709778, -0.6323279142379761, -2.1481120586395264, 0.1052125096321106, 0.20467950403690338, -1.0856108665466309, -1.1932661533355713, 1.0086171627044678, -0.6074013113975525, 0.8798360228538513, 0.7332020401954651, -0.27729836106300354, -1.0219091176986694, 0.16480927169322968, 1.66152024269104, 1.721445083618164, 1.6284186840057373, 0.36151352524757385, 0.2824908196926117, 2.7158405780792236, 0.7520208954811096, -1.2497179508209229, 1.0656929016113281, -0.19088926911354065, 1.129960298538208, 0.023673754185438156, 0.5525356531143188, 0.9953484535217285, 0.9551035761833191, -0.26260805130004883, 0.74855637550354, 2.1078405380249023, -0.5702523589134216, -0.32885271310806274, 1.84063720703125, 0.8388649821281433, -1.3325855731964111, 1.1000893115997314, -0.7121372818946838, 0.29523664712905884, -1.1743897199630737, 0.486172616481781, -1.8659775257110596, -0.5008713006973267, 0.4891637861728668, -0.8721544742584229, 1.0388062000274658, 0.12429133802652359, 0.730502724647522, -1.1104732751846313, 0.11218062788248062, -1.3088719844818115, 0.35930943489074707, 0.17135697603225708, 1.371766448020935, -0.3360561430454254, -0.697975754737854, 0.4061111807823181, -0.4913215935230255, -2.6151509284973145, 0.6802306175231934, -1.369504451751709, 0.2550773024559021, 1.0843136310577393, -1.4028940200805664, 0.9826677441596985, -0.8948073387145996, 0.4644332230091095, 1.0426470041275024, -0.6541614532470703, -0.8122471570968628, -0.6662106513977051, -0.2943810522556305, 0.3288908302783966, -0.3734895884990692, -0.3333844542503357, 0.7288069725036621, -0.4039565920829773, -1.9203373193740845, 1.334786295890808, -1.442900538444519, 1.1315957307815552, -0.3431835472583771, -0.3678378164768219, 0.18570533394813538, 0.7525415420532227, 0.11021207273006439, -0.43588921427726746, -0.9451606869697571, 0.8661389350891113, 0.3941035568714142, -0.5881394743919373, -0.8054935336112976, -0.13211487233638763, 0.6359344124794006, 0.38120824098587036, 0.3652251660823822, 0.08373499661684036, -1.834027886390686, -0.4476364254951477, 1.0388553142547607, -0.3011203408241272, 0.8094319701194763, -0.8141251802444458, 0.05414367467164993, 1.0556906461715698, -0.4100395143032074, 0.30196699500083923, 0.32856011390686035, 0.4511697590351105, -0.25599804520606995, -0.27411437034606934, 0.5912224054336548, 0.920599639415741, -1.6850457191467285, -0.2536361515522003, -0.14800430834293365, 0.5182045102119446, -1.2518476247787476, 1.749191164970398, 0.9070852398872375, -0.7899767160415649, 0.804038405418396, 0.36287370324134827, 1.1080939769744873, -1.2974166870117188, 0.27529340982437134, 0.4113529324531555, -0.9921384453773499, 0.3149382770061493, -0.2423294335603714, 0.5072618126869202, 0.10774585604667664, 0.8118505477905273, 1.777259111404419, 0.04520207643508911, -1.2136712074279785, -0.3640434145927429, -0.3475940525531769, 0.3704054057598114, -0.35642334818840027, -0.3443593382835388, -1.364614486694336, -0.4196704924106598, 2.2630136013031006, 2.577143907546997, 0.9272836446762085, 1.7549512386322021, -2.1358697414398193, -0.012433580122888088, -0.46119424700737, -0.7195416688919067, 0.2784026563167572, -0.4620669186115265, -0.8978807330131531, 1.390667200088501, 0.2967715263366699, -1.1863383054733276, -0.07996895164251328, -0.5723153948783875, 0.38209226727485657, -0.4172542095184326, 0.36302053928375244, -0.16628730297088623, 0.9910991787910461, 2.098548412322998, 0.2420828640460968, -1.7207942008972168, 0.04039393365383148, 0.19066563248634338, -0.7183220982551575, -2.144895315170288, 1.4780157804489136, -0.9851848483085632, -0.841195821762085, 0.6702216863632202, -1.54560124874115, 0.9782780408859253, 0.6589878797531128, -0.5656788349151611, -1.3140088319778442, 0.9751495718955994, -1.732364535331726, 0.967926561832428, 0.7758026123046875, -0.26182419061660767, -0.21302290260791779, 1.451125144958496, -1.6014201641082764, -0.7688959836959839, 0.5583451390266418, 2.054441213607788, -0.25634071230888367, 0.04250846803188324, 1.6281931400299072, -0.3476887345314026, -1.229854941368103, -1.3383196592330933, -1.6366775035858154, 1.0157850980758667, -0.10904646664857864, 0.04702053964138031, -2.180018424987793, -0.25444087386131287, 0.06308010965585709, -1.2997788190841675, -0.3205750286579132, 1.4686123132705688, 0.7904533743858337, -0.3752448558807373, 0.36371734738349915, 2.112987756729126, -0.4794014096260071, -2.242025852203369, 2.111802101135254, -1.3868274688720703, 0.12889041006565094, 0.8719761967658997, -0.5762218236923218, 0.0938190221786499, 1.7925091981887817, 0.08907026797533035, 1.1930817365646362, -0.23521044850349426, -0.29561635851860046, 0.41521427035331726, 3.4762864112854004, -0.1259005069732666, -0.15487319231033325, 0.9490741491317749, 0.5081833004951477, 0.8749446868896484, -0.6771740913391113, 2.203378677368164, 0.3934682309627533, 0.8466801047325134, -0.1656523197889328, -0.08691700547933578, 0.1892976611852646, 1.5587254762649536, 0.09595894068479538, 0.20202846825122833, 0.3312011659145355, 0.6326613426208496, -1.6893329620361328, 0.28183838725090027, 0.29814431071281433, 0.10119707882404327, 1.5953266620635986, 1.4434444904327393, 1.0982478857040405, -0.3648030459880829, -0.2507616877555847, 0.7515501379966736, 0.5913482904434204, 0.9948733448982239, -0.04997723549604416, -0.2466902881860733, -0.733055830001831, 0.6788303256034851, -0.6316507458686829, -0.19888393580913544, 0.8637853264808655, 2.149989366531372, 0.28157368302345276, -1.6109732389450073, -0.9020058512687683, 0.4965970516204834, 1.7452316284179688, -2.0167837142944336, -1.5460542440414429, 0.6717044711112976, -0.02397678978741169, -1.1305499076843262, -0.3429587781429291, -0.8319064974784851, -0.7731413841247559, -0.0667681023478508, 0.04294585809111595, 0.22578281164169312, -0.506496250629425, 0.2871968746185303, -2.394967794418335, 1.3757516145706177, 0.0730324536561966, -1.090097188949585, -1.1916059255599976, -0.5354290008544922, -0.054442036896944046, -1.797329306602478, 0.0053952536545693874, -1.4058588743209839, 2.6889069080352783, 0.7361181974411011, -1.7398992776870728, 1.1418370008468628, 0.2707785367965698, 0.6302252411842346, 0.2754272222518921, 2.046377658843994, -1.6604119539260864, 0.8430006504058838, 1.5021133422851562, 1.3893076181411743, -0.8173156976699829, -0.3937581777572632, 1.7432143688201904, 0.29676416516304016, -0.8033289313316345, 1.9284683465957642, -0.13688698410987854, 1.6136958599090576, 0.32853785157203674, 1.6417083740234375, -1.277596354484558, 1.0631777048110962, 0.48607394099235535, 0.7842242121696472, 0.5278948545455933, -0.4500100612640381, 0.7266091704368591, -0.5074071884155273, 0.3195776045322418, 0.7546372413635254, -0.14094319939613342, -0.11539371311664581, 0.058976538479328156, 0.5381873846054077, 0.24254512786865234, -0.5356466770172119, 0.5196315050125122, 0.43468084931373596, 2.1866862773895264, 0.4444039463996887, -0.6150646209716797, 0.08712106943130493, -0.7822000980377197, -0.9907785654067993, 1.1617733240127563, -0.41803911328315735, 1.5017657279968262, 1.121272325515747, -0.7547435760498047, 0.7823238372802734, -0.24861879646778107, -0.28388693928718567, 1.522696614265442, 0.06811509281396866, -0.16706475615501404, 1.0332683324813843, 1.4757411479949951, -0.1671185940504074, 2.628394365310669, 0.5527686476707458, -0.25521519780158997, -0.2751752734184265, -2.568321704864502, -1.7237366437911987, 1.1046478748321533, -0.9768021106719971, 0.8872721791267395, -0.6293426156044006, 0.8676963448524475, -1.6633628606796265, -0.9491488337516785, -2.0108048915863037, 0.5326153039932251, 0.3057022988796234, -1.414623498916626, -0.7633950710296631, 1.1604076623916626, -0.4988524913787842, -0.40808045864105225, 1.063213586807251, 1.798488736152649, -0.3252601623535156, 0.63627690076828, 0.165640726685524, 1.0675649642944336, 0.594936728477478, 0.1519857943058014, 0.20195789635181427, -0.37614649534225464, -0.20946095883846283, 1.0301165580749512, -1.8566147089004517, 0.8144969940185547, 1.0537354946136475, 1.9329795837402344, -0.4268084764480591, 0.7275763154029846, -2.2140660285949707, 0.16113202273845673, -1.218000054359436, -1.4765901565551758, 0.5577021837234497, 1.15093994140625, -1.9407087564468384, 1.0140870809555054, -1.7377971410751343, -0.41131991147994995, 0.6197963356971741, -1.492668628692627, -0.9398611783981323, 1.3518645763397217, -0.10799030214548111, 0.1427435427904129, -1.5298429727554321, -0.8296457529067993, 1.2907302379608154, 3.037945508956909, -1.6265220642089844, 1.7763313055038452, -0.179753378033638, -0.32133010029792786, -0.5293676853179932, 0.866566002368927, -0.5172079801559448, 0.7691351175308228, -0.5179702043533325, 2.717554807662964, -0.8604112267494202, -0.1435786634683609, -0.7514405250549316, 0.04866556450724602, 0.7596191763877869, 0.2208714336156845, -1.299664855003357, 0.52178555727005, -1.268270492553711, -0.40370973944664, 0.08050229400396347, 0.7612770199775696, -0.7889304161071777, -0.15910759568214417, -1.6244515180587769, 0.26140880584716797, 1.7580689191818237, -0.8273977041244507, 0.4604659676551819, 0.11492015421390533, -0.6339196562767029, -1.5008524656295776, 0.6816296577453613, -0.7624927759170532, -0.051179155707359314, -0.14435215294361115, 1.0705910921096802, -0.45138019323349, 0.23381724953651428, 0.6558300256729126, 0.8526448607444763, 1.198560357093811, 1.2001547813415527, 0.38426581025123596, -0.6824154853820801, -0.37937262654304504, 0.5637239813804626, 1.9434003829956055, -0.2424890697002411, 0.7040741443634033, -0.4641372859477997, 0.9865248203277588, -0.21193289756774902, 0.9314664602279663, -2.741809129714966, -0.5945034623146057, -0.11985667049884796, 1.1678568124771118, -1.64409601688385, -0.7846600413322449, -0.33514299988746643, -0.44796183705329895, 0.4264882802963257, -0.002446261467412114, 0.5663501620292664, -0.11138823628425598, 0.27821487188339233, 0.4350475072860718, -0.39198222756385803, 2.121147632598877, -0.1022077426314354, 0.031958311796188354, -1.266676425933838, -0.6725632548332214, 0.3404234051704407, -0.8520134687423706, -0.7090504169464111, 0.0025470093823969364, 0.25472256541252136, 0.17000073194503784, -0.8726799488067627, -0.33280423283576965, 0.14929918944835663, -0.03821125254034996, 0.9580556750297546, 0.5623599886894226, 0.28105148673057556, 1.6048767566680908, -0.5232256650924683, 0.23356011509895325, 1.0014492273330688, -0.6746131181716919, 0.22193032503128052, 0.03761275112628937, 0.48174381256103516, -2.0031073093414307, -1.7431241273880005, -0.11686082929372787, 0.41251441836357117, -0.24724842607975006, 0.25769802927970886, 0.09751676023006439, 0.7307063937187195, 1.6665551662445068, 1.3913692235946655, -0.5222102999687195, -0.7657800912857056, 0.1673000156879425, -0.10970575362443924, 1.3018345832824707, -0.5133805274963379, 0.43924522399902344, 1.8297038078308105, 0.8547548651695251, -0.7367083430290222, -0.5830478668212891, -0.9122246503829956, -0.20021766424179077, -0.6186102628707886, -1.3094948530197144, -0.654104471206665, -0.0589149035513401, 1.545866847038269, 1.0731781721115112, -0.1665317565202713, -1.2781174182891846, -0.5656571984291077, -0.05337805673480034, 0.9021354913711548, 0.2151806503534317, -0.7010339498519897, -1.2593777179718018, -0.5959608554840088, 0.9137773513793945, -0.4845891296863556, -0.7013125419616699, 0.11802979558706284, -1.4883557558059692, -1.2410975694656372, -1.2392725944519043, 0.7984689474105835], [0.31554391980171204, 0.9468426704406738, -0.8830378651618958, 0.8284426331520081, 0.136955127120018, 1.488985300064087, -0.7388644218444824, -0.6491047143936157, 0.17833459377288818, 0.47410938143730164, -2.2968454360961914, -0.43769973516464233, -1.7651736736297607, 1.1861743927001953, -0.4652822017669678, 0.8927371501922607, -0.037166304886341095, 0.25216639041900635, -0.5513513088226318, -0.7857797145843506, 1.659864068031311, -0.18233664333820343, 0.39794859290122986, -1.9022200107574463, -0.6656777858734131, -0.023631758987903595, -0.8095428347587585, 0.12861227989196777, 0.34210219979286194, -0.07309836149215698, -0.09787286072969437, 0.9709421396255493, 0.26299744844436646, 1.0697087049484253, -0.05338270589709282, 0.09092129021883011, -0.31868216395378113, 1.3910770416259766, 1.4244316816329956, -2.160033702850342, -0.9991474151611328, -0.258588045835495, 0.0057619716972112656, -0.99570631980896, 0.45142245292663574, -1.520624041557312, 0.17143529653549194, 0.7677675485610962, -0.9681156277656555, 0.32979971170425415, 0.3484462797641754, -0.6855602264404297, -1.5826809406280518, 0.5110569596290588, 0.8519154787063599, -0.9291814565658569, -0.5647756457328796, 1.4586734771728516, 0.159803107380867, -0.7350066900253296, 0.6692747473716736, 0.7507075667381287, -0.05708921328186989, -0.4280475676059723, 1.7962417602539062, 2.2327468395233154, -0.25897371768951416, -0.35664626955986023, -1.5048538446426392, 0.23163555562496185, 1.0556731224060059, -0.3815023601055145, 0.06131251901388168, 0.24552568793296814, -0.14674413204193115, 1.9693007469177246, 1.0161932706832886, 1.116250991821289, 0.27457332611083984, -0.14527162909507751, -0.995890736579895, -0.6916118264198303, 1.1242135763168335, -2.421750545501709, -1.5776005983352661, -1.3056045770645142, 1.037603497505188, -1.8304988145828247, 0.23551520705223083, 0.4091241657733917, 0.9162877798080444, 0.8809372186660767, -2.3516101837158203, -0.2075832337141037, -1.4430575370788574, -1.1687290668487549, 0.5223075747489929, 0.7465071678161621, 0.23282070457935333, -1.5503875017166138, -1.3646827936172485, -0.37321484088897705, 1.746181845664978, 0.31832894682884216, -0.39324527978897095, 0.8322449326515198, -0.8040297031402588, 0.04969911649823189, -1.8923768997192383, 1.2877376079559326, -0.35724788904190063, -1.6619832515716553, -0.5328699946403503, 1.4368040561676025, -0.32700875401496887, 1.3995577096939087, -0.8846505880355835, -0.09336654096841812, -0.7694682478904724, 0.13101667165756226, 1.078288197517395, 0.8340743780136108, -0.47437551617622375, -0.2614520192146301, -1.9379193782806396, 2.0389342308044434, -0.5991170406341553, -0.12167388200759888, -1.5427978038787842, -0.23209834098815918, 0.9040617942810059, -0.4014490842819214, -0.21197807788848877, -0.5975459218025208, -0.33597081899642944, -1.1447983980178833, -1.3256587982177734, -0.3252386450767517, -0.12685224413871765, 0.7870436906814575, 0.3070021867752075, 0.36867520213127136, -1.1803098917007446, 1.273692011833191, -1.9210131168365479, -1.0431503057479858, -0.4630296230316162, 0.26647889614105225, -0.33906078338623047, 0.11280995607376099, -1.2182315587997437, 0.22607611119747162, 1.943016767501831, -0.6560066938400269, 0.663671612739563, 0.19578337669372559, 0.2141553908586502, 0.8851134181022644, -0.3398485481739044, -1.0716124773025513, -0.25472745299339294, -0.11258123070001602, -0.5137408375740051, 1.1248599290847778, -0.9709022045135498, 0.8921418190002441, -0.2398674488067627, 0.48073333501815796, 0.9117143750190735, 1.0575107336044312, -3.2064976692199707, -0.8013575673103333, -1.293513298034668, 1.0947182178497314, 1.5768985748291016, 1.2635958194732666, 1.7429033517837524, 0.10081982612609863, -0.2539592683315277, 1.2732837200164795, 1.600521445274353, 1.3886938095092773, -0.5355854630470276, 2.4187633991241455, 1.1167036294937134, -0.4961231052875519, -0.31141117215156555, 0.5952923893928528, -0.5958914756774902, -0.9133459329605103, 0.877737283706665, -1.599347710609436, -0.9322870969772339, -0.0582866296172142, 1.1996772289276123, -0.7091364860534668, 0.23139382898807526, -0.5633083581924438, -0.11538947373628616, -0.9481534361839294, 0.3436405658721924, 1.8192681074142456, 0.8679064512252808, 1.1003366708755493, 0.2651461362838745, 0.35670116543769836, 0.7407255172729492, 0.9448110461235046, -0.11387968808412552, -0.17192991077899933, -0.8671402335166931, -0.10823000222444534, 2.1183667182922363, -1.8079367876052856, 2.2161967754364014, 1.593867540359497, -2.7410383224487305, -0.5156766772270203, -0.054862819612026215, 0.11492131650447845, -1.0338085889816284, -0.3393929898738861, -0.39500144124031067, 0.028202610090374947, -0.740882396697998, 0.11864203959703445, -1.033455729484558, -0.606285035610199, 1.2743220329284668, -0.6586076617240906, -0.44177335500717163, 0.43533194065093994, 0.015504959039390087, -1.1845611333847046, 1.0617772340774536, 0.4900909662246704, -1.505929946899414, -0.462179958820343, 0.42081418633461, 0.38567736744880676, 0.24205230176448822, -1.837939739227295, -1.7343469858169556, -0.45971354842185974, 1.0021538734436035, -0.7331430912017822, 0.8316192030906677, 2.061011791229248, 2.3243417739868164, 0.6308802962303162, -1.4975242614746094, 0.3016745448112488, -1.082338571548462, -0.9440082907676697, -0.155913308262825, 1.837812066078186, 0.36289089918136597, -1.5402941703796387, -0.39006030559539795, -0.6318806409835815, -2.22224760055542, 0.47999611496925354, 0.38000649213790894, 1.3545563220977783, -1.174494981765747, 0.764981746673584, -0.875323474407196, 0.5910536050796509, 0.7564129829406738, -1.2085132598876953, -0.2859526574611664, 0.4446289837360382, -2.283747673034668, 0.051590222865343094, -0.8568520545959473, 0.0632425919175148, 0.3475251793861389, 0.10353852063417435, 0.43317878246307373, 1.8738268613815308, -1.2384618520736694, -0.399943470954895, -0.9083004593849182, 0.5433780550956726, 0.6585468053817749, -0.07590151578187943, 0.4321730434894562, 2.1644446849823, -0.4938429892063141, 1.2393012046813965, -1.2781257629394531, -0.4022432565689087, 0.7843387126922607, 1.1264792680740356, -1.078310251235962, 0.02182663232088089, 1.2843462228775024, -1.2786853313446045, 0.25506651401519775, -1.8808284997940063, 0.8921090364456177, 1.6744407415390015, 2.393005132675171, -1.604131817817688, 0.09003423899412155, -1.0195084810256958, 0.9179377555847168, 0.3856310546398163, -2.0949976444244385, 0.48096200823783875, -0.06216742843389511, -0.428718239068985, -0.25171369314193726, -0.2935560345649719, 2.033046007156372, 0.6438406705856323, 0.38484734296798706, -1.3699344396591187, 0.11006148904561996, -0.8952012062072754, 0.17701943218708038, 1.2713321447372437, 0.4722996950149536, -0.45923930406570435, -0.11232075840234756, -0.39622220396995544, -1.9527206420898438, 0.7855361700057983, 0.3294805586338043, 0.46605685353279114, -1.0907553434371948, 1.5826433897018433, 0.5332044363021851, 0.5810602903366089, 0.7189028859138489, -0.8376399874687195, 0.6391209363937378, -0.37617242336273193, -0.2525956332683563, -0.13479956984519958, 0.04619400203227997, 1.3681633472442627, -0.6159574389457703, 0.8558407425880432, -0.573509931564331, -0.5045598149299622, 0.6479371190071106, 0.9178072810173035, 0.31749704480171204, -0.09263033419847488, -1.2155542373657227, -0.9863889813423157, 0.015714041888713837, -0.3114827573299408, -1.266000509262085, 0.16501711308956146, -1.105283498764038, -0.9872756600379944, 1.7181370258331299, -0.660711407661438, 2.6585042476654053, 0.6564587354660034, 0.025715772062540054, -0.28835245966911316, 0.094589464366436, -0.7648882865905762, -1.3453209400177002, -0.4295223355293274, 0.5803061127662659, 0.6217212080955505, 1.0521981716156006, -0.12201744318008423, 0.49859026074409485, 0.506993293762207, 0.18494631350040436, 1.256819248199463, -1.4753668308258057, 0.5394296646118164, 0.2624501883983612, 0.9510796070098877, -1.0524203777313232, 0.1056378185749054, 2.3293251991271973, 0.3080098628997803, -1.6129347085952759, -1.4058585166931152, -0.9359148740768433, -1.6561203002929688, 0.749718964099884, 1.5289684534072876, 0.6789828538894653, -2.2341153621673584, 0.21989914774894714, 0.2549233138561249, -1.4092966318130493, 0.2164895087480545, 1.2242093086242676, 1.9482406377792358, 0.24466530978679657, -1.750646710395813, 1.0697813034057617, -1.8210550546646118, 2.108931064605713, 1.6491843461990356, -0.7695111632347107, 1.0723658800125122, 0.16722635924816132, -1.0504778623580933, -0.046997882425785065, -0.6671862006187439, -0.6908284425735474, -0.4831008017063141, 0.3596813380718231, 0.6463512182235718, -0.2651107609272003, 1.8518894910812378, -1.1865345239639282, 0.9535281658172607, -0.727039098739624, -1.1069337129592896, 0.49106892943382263, -0.8793331980705261, 1.7417943477630615, -1.2573304176330566, -0.04099566861987114, 0.3682704269886017, -0.2784186601638794, -0.7464390397071838, -0.15796306729316711, -1.4308812618255615, 1.139412522315979, 0.9068099856376648, -0.8809203505516052, -1.5435420274734497, 0.06391269713640213, 0.6281440258026123, -1.1086043119430542, 0.05128875747323036, 1.2515977621078491, -2.9073872566223145, -0.5443276762962341, -0.5303800702095032, -1.5551444292068481, -0.17312681674957275, -0.50460284948349, 0.15169179439544678, 0.059047091752290726, -1.177056908607483, -0.4292265772819519, -0.34974515438079834, -0.3232286274433136, -0.3299967348575592, 0.469509094953537, -1.3344169855117798, 0.7020288705825806, 0.9029404520988464, -0.6518291234970093, 1.6800376176834106, -0.21844962239265442, 1.3254932165145874, 1.076625108718872, 0.4272724986076355, 1.7602558135986328, 0.3170839548110962, 1.2172939777374268, -0.6714222431182861, -0.42719951272010803, 0.8035402894020081, -0.07045416533946991, 0.0742444321513176, 1.61826753616333, 0.4665381908416748, -0.4618222713470459, 0.4881229102611542, -1.946253776550293, -0.6875525712966919, 0.1443738043308258, 0.09304308891296387, -0.9899500012397766, 0.33477649092674255, 0.10263314843177795, -0.2252224087715149, 0.07614139467477798, 0.06060327962040901, -0.16685329377651215, -0.2411632239818573, 0.47336217761039734, 0.15553458034992218, 0.4001275897026062, -1.083086371421814, -0.8431837558746338, 1.5595344305038452, -1.3108453750610352, -0.30841773748397827, -0.6274085640907288, 0.06433815509080887, -2.3860278129577637, 0.47313034534454346, -0.47404852509498596, -0.6467100977897644, -1.348181128501892, 0.07038617134094238, -0.469765841960907, -0.0847405418753624, -0.8749352097511292, -0.1166098490357399, 0.8582924604415894, -0.33289188146591187, -0.15080483257770538, 0.7836959958076477, -1.0128847360610962, 0.21751827001571655, 2.093113660812378, 1.2566272020339966, -1.8875889778137207, -0.8817445635795593, -1.7500797510147095, -1.3140990734100342, -0.4800059199333191, 0.23941051959991455, -0.23062647879123688, -1.0153475999832153, 0.5301973819732666, -0.9177370667457581, 0.5513293147087097, 0.6975772976875305, -0.39586350321769714, 0.6959956884384155, -0.7463005185127258, 0.02840990386903286, 1.1696771383285522, 2.1100127696990967, -0.7563245892524719, 2.424335241317749, -0.588435709476471, -0.33678138256073, -0.5864009857177734, 0.5719363689422607, 0.5508580803871155, 0.08794810622930527, -1.8096508979797363, -0.08303078263998032, 0.5526244044303894, 1.2620165348052979, -0.13661642372608185, 0.08195402473211288, -1.0759520530700684, 0.6901183724403381, 0.5740164518356323, 1.1531637907028198, 1.0599819421768188, 0.006271523889154196, 0.06871650367975235, -1.911016821861267, -0.6936699151992798, 0.013932428322732449, 0.23069268465042114, 0.7400948405265808, 0.2067122459411621, -1.4352091550827026, -1.4426742792129517, 1.3822625875473022, 0.9972063302993774, 1.1206830739974976, -1.5573238134384155, -0.6257064938545227, 0.648329496383667, 0.2414024919271469, -0.6772330403327942, 1.0121526718139648, -0.5055499076843262, -0.3668789267539978, 0.004235758446156979, -0.631417453289032, -0.4362949728965759, 1.414790153503418, -0.8045867085456848, 1.0859758853912354, 2.8574254512786865, -0.27026161551475525, -1.2387921810150146, -1.1246483325958252, 0.18523603677749634, 1.2215015888214111, 0.34843558073043823, 0.7496674656867981, -1.6908591985702515, 0.14628569781780243, 0.3060477674007416, 0.9419485926628113, -0.15754371881484985, 0.659187376499176, 0.19939030706882477, -0.5759878158569336, 0.4174468517303467, 0.16695111989974976, -1.3385515213012695, -1.455737829208374, -1.3192075490951538, 0.711759090423584, 0.3822349011898041, 0.6253607273101807, -0.23279327154159546, 0.25679394602775574, 0.9099082350730896, -0.5520229339599609, 2.197511672973633, -1.9862555265426636, 1.1388949155807495, 0.053883783519268036, 2.036569833755493, -0.3902525305747986, 0.3057425022125244, 0.7204521298408508, 1.9882853031158447, 1.4033873081207275, -2.693039655685425, 1.3428622484207153, 1.3027818202972412, -1.4939700365066528, -0.4124404489994049, -0.328717976808548, 0.24744020402431488, -0.40576332807540894, -1.3262081146240234, 0.9726631045341492, -1.606933832168579, 1.4022150039672852, 1.0618902444839478, 1.9704424142837524, 0.5152198672294617, -1.4918900728225708, -0.9802152514457703, 0.3482568860054016, -0.0838485136628151, 1.5670320987701416, -0.829048752784729, -1.6878377199172974, -0.7057941555976868, -1.1527425050735474, -0.7225295305252075, 1.2106515169143677, -1.1601073741912842, 1.2492940425872803, -1.4984519481658936, -0.9468695521354675, 0.09289300441741943, 0.8349572420120239, -0.10736404359340668, -1.628845453262329, 0.06693503260612488, -1.4667140245437622, 0.01258990541100502, 2.0955429077148438, -0.7768795490264893, 1.1161775588989258, 0.34783387184143066, 0.07662123441696167, 2.165792942047119, 0.9231845736503601, -0.6269224286079407, -0.1797071099281311, 0.21752497553825378, -0.5586821436882019, -0.7380266785621643, -0.015199937857687473, 0.17460836470127106, -0.46197274327278137, -0.21872562170028687, -1.4266268014907837, -0.5231509804725647, -0.30140990018844604, -0.7947575449943542, -1.3083397150039673, 2.6558218002319336, -0.24484975636005402, -0.28970226645469666, -0.3630060851573944, -0.27570414543151855, 2.143739700317383, 0.4531739056110382, -2.3759942054748535, 0.13860341906547546, -1.164106845855713, -1.5808707475662231, 1.3147542476654053, 0.6828218102455139, -0.03320452570915222, -1.1896895170211792, -0.5470660924911499, 0.5344218611717224, -0.6817238330841064, 0.5519928932189941, 0.4408431351184845, 1.5030419826507568, -0.36551541090011597, 0.42292341589927673, -1.956364393234253, 0.3626849055290222, -0.307726114988327, -1.2559170722961426, 0.195190891623497, -0.8016663789749146, -0.8552541732788086, -0.5959177613258362, 1.2526224851608276, 1.9504083395004272, 0.5983608961105347, 1.601159691810608, -1.138224482536316, -1.0906776189804077, -0.6535196900367737, 1.4884450435638428, 0.14237193763256073, -0.6297393441200256, -0.5057768821716309, 0.11074049770832062, 1.4686810970306396, 1.8057321310043335, -0.6564664840698242, 2.9759507179260254, -0.07474750280380249, 1.300846815109253, 0.5745736956596375, -0.7446647882461548, -0.14558127522468567, -0.430113285779953, 0.30190590023994446, 0.7423702478408813, -0.21507160365581512, -1.840659260749817, 0.7417845129966736, 0.5329802632331848, -1.674854040145874, 1.574510097503662, 0.06264379620552063, -3.1041996479034424, -0.9069308042526245, 0.45163366198539734, -0.8149911761283875, 0.28316807746887207, -0.34336555004119873, -0.5720546245574951, -0.28771209716796875, 0.3847943842411041, -0.7367004156112671, -1.8508002758026123, -0.9272525906562805, -1.5288610458374023, -0.14260278642177582, -0.4434899389743805, -0.9159780740737915, -0.2637861669063568, 1.084283471107483, 0.7586524486541748, 0.976061999797821, 0.1854962855577469, -2.5056445598602295, 0.23792950809001923, 1.1163194179534912, -0.6852752566337585, -1.8668190240859985, -1.22272789478302, -1.9294490814208984, 2.340996026992798, -3.0488474369049072, 0.8703863024711609, -0.16895875334739685, 0.09513223171234131, 1.8537367582321167, 0.526969313621521, -1.3148061037063599, 0.0968453586101532, 0.5500711798667908, -1.1904902458190918, -0.8644667863845825, 0.49426668882369995, -0.5485051274299622, 0.45003145933151245, -0.6817846894264221, -2.3191184997558594, -1.1379770040512085, 0.2183881551027298, -0.15510962903499603, -0.07924798130989075, -0.3938157558441162, 1.0815693140029907, 0.5872927308082581, -0.2485334724187851, -0.11835753917694092, -0.691523015499115, -3.4480998516082764, -0.21529336273670197, -0.34339290857315063, -0.16691407561302185, -0.6311558485031128, 0.9506082534790039, -0.9337201118469238, -0.17528799176216125, 0.9032415747642517, 0.566257655620575, 0.20599107444286346, 0.22580276429653168, 0.2606307566165924, -1.101034164428711, -0.5683016180992126, 0.6031249165534973, 0.8095306158065796, 1.3414812088012695, -0.046375591307878494, 1.395693063735962, 1.390212059020996, -1.4204895496368408, -1.5807085037231445, 1.8359979391098022, 0.11491553485393524, -0.7636561393737793, 1.2387869358062744, -0.24668067693710327, 0.6810582280158997, -1.052632451057434, -1.377017617225647, 1.223473310470581, -0.1378900110721588, -0.6352535486221313, 0.4912758767604828, 1.0870046615600586, 0.4636450707912445, -0.8030445575714111, 1.1170183420181274, -0.9460020065307617, 2.080054521560669, -0.7074682116508484, -1.0836107730865479, -0.057108357548713684, 0.7750661373138428, 1.6236279010772705, -0.32136011123657227, 0.5687274932861328, 2.02443790435791, 0.9478681683540344, 0.7359470725059509, 1.1608120203018188, -1.3882167339324951, 0.24639685451984406, -0.9890652298927307, 1.6332588195800781, 0.7500525712966919, 2.0515570640563965, -1.279140591621399, 0.716930627822876, -1.450510859489441, 0.4357961118221283, 0.7327476739883423, -0.36585289239883423, 1.369217872619629, 0.4156225323677063, -0.850314199924469, -0.21452674269676208, -1.661110758781433, -0.5790983438491821, 1.0842533111572266, -0.9940789341926575, -0.418033242225647, 0.4672284722328186, -0.4990542232990265, 0.5620236396789551, -0.7382903695106506, -1.8391344547271729, -1.442937970161438, 0.821840763092041, -0.30023351311683655, 0.577729344367981, 1.584143877029419, -1.4459751844406128, 0.7409907579421997, -0.6127356290817261, 0.02482500672340393, -0.621910810470581, -0.9688860774040222, 0.41022875905036926, 0.03852434456348419, 0.7308151125907898, -0.7317889332771301, -0.5632158517837524, -0.2604653239250183, 0.3242385983467102, 1.88881516456604, 0.36710765957832336, 0.8451212644577026, 0.2483418583869934, -2.629782199859619, 0.0380437895655632, 1.7220300436019897, 2.1011197566986084, -1.5328940153121948, -0.5330896377563477, 0.9670285582542419, 1.2435086965560913, 0.04304662346839905, 0.1499803215265274, 0.14683735370635986, -0.33094924688339233, 0.41808798909187317, 0.9448599815368652, -0.5286858677864075, -1.0087286233901978, 1.1884238719940186, -0.323990136384964, -1.1334385871887207, -0.48074936866760254, -1.2307511568069458, 0.11630988121032715, 1.182708740234375, -0.1135418638586998, -0.18554338812828064, -0.6721420288085938, 0.9184836149215698, 0.4448665976524353, -0.2803179621696472, 0.05215895548462868, 0.12104175984859467, 0.8081079125404358, 0.9189141392707825, -0.09234458953142166, -0.06368717551231384, 0.4685686528682709, 1.1492568254470825, -0.7525709271430969, -1.114190936088562, 0.4957486689090729, -0.5207264423370361, 0.031209899112582207, 0.18286201357841492, 0.5048130750656128, 0.20176486670970917, -0.2075638473033905, -0.5831232666969299, 0.9994485378265381, 1.223475694656372, 1.053293228149414, 0.8282904624938965, 1.560300350189209, 0.22816966474056244, -0.9811561107635498, 1.2075715065002441, -1.2550791501998901, -0.7162761092185974, 0.798403799533844, -2.5923659801483154, -0.2957842946052551, -0.5017287135124207, 0.11746174842119217, 0.8527575135231018, 2.5550153255462646, 0.40107184648513794, -1.1239194869995117, -0.9323769211769104, 1.4654101133346558, -1.2559140920639038, 0.08631043136119843, -2.4773001670837402, -0.8010255694389343, 0.32331863045692444, -0.48740777373313904, -0.5958675742149353, 0.4767318069934845, -1.4134361743927002, 2.049574375152588, 0.041588157415390015, 0.18246731162071228, -0.45541268587112427, -0.4283807873725891, -0.4815029203891754, 0.5836601853370667, 0.32665619254112244, 0.42841649055480957, 0.4423862397670746, -0.8251342177391052, 0.8870680332183838, -0.6709591746330261, 1.3380604982376099, 0.9313718676567078, 0.4942840337753296, 1.4853925704956055, -1.7432525157928467, -0.8579964637756348, 1.3474594354629517, -2.058919668197632, -0.8431930541992188, 0.42450785636901855, -0.36368072032928467, -0.03422187641263008, -1.223649024963379, 0.3061884939670563, 1.9773706197738647, -0.09545329213142395, 0.2697286307811737, -0.10865942388772964, -0.20507071912288666, -0.3529006838798523, 2.2130825519561768, -0.07034336775541306, 1.278922200202942, 0.7697725296020508, -0.5915616154670715, 1.3088443279266357, 0.056950218975543976, -0.3346659243106842, -1.1696341037750244, -0.5834382772445679, -1.206398367881775, -0.29807397723197937, 0.2748185098171234, 1.8500113487243652, -0.1492954045534134, -0.7858622670173645, 0.12417047470808029, -1.5629665851593018, -0.07073038071393967, 0.7098648548126221, -2.4259376525878906, 1.292488694190979, 1.7135634422302246], [0.43088650703430176, 0.9015074372291565, 0.03483056649565697, 0.07431229203939438, -0.4803886115550995, -0.9523977041244507, 0.9428722858428955, -1.018609881401062, 1.4044766426086426, 0.33039242029190063, 0.8795170187950134, 0.7068949937820435, 0.15214042365550995, -1.7935233116149902, 0.8012986779212952, 2.2116787433624268, -0.3590102195739746, -0.7087002992630005, -1.5340927839279175, -1.7160757780075073, 1.6445525884628296, -1.5032880306243896, -0.5175577402114868, 0.4245374798774719, 0.891806423664093, 0.2350376546382904, 0.53173828125, 1.877346158027649, -0.07702624052762985, 0.5159480571746826, 0.42965662479400635, 0.8803831934928894, 0.562566876411438, 0.3870014548301697, -0.801862359046936, 0.14725929498672485, -0.6851588487625122, 0.3100179135799408, -0.3743037283420563, -0.1543578803539276, -0.2828747034072876, 0.23952937126159668, 0.13954506814479828, 0.23312242329120636, -0.3703012764453888, -0.8013951182365417, 1.8555692434310913, -0.18208996951580048, 0.8420194983482361, 0.9653210043907166, 0.6853101253509521, 0.6586096882820129, 1.9502217769622803, -0.5973750352859497, -1.898499608039856, -0.7066099047660828, 0.9027326703071594, -0.695442259311676, 1.1973092555999756, 0.31495437026023865, 0.6352695226669312, -1.0038479566574097, 1.178568720817566, -1.3543658256530762, 0.6569904088973999, -0.5130417346954346, 0.5720847249031067, 0.996360719203949, -0.7987593412399292, -1.1008368730545044, -1.174012541770935, -0.3917543888092041, -1.3817858695983887, 1.159496784210205, -1.3176921606063843, 0.5200196504592896, -1.039259910583496, -0.743175745010376, -1.2328088283538818, -1.731695532798767, 1.7099380493164062, 0.5177154541015625, 0.3109472095966339, 1.5196915864944458, -0.7040455341339111, -0.33393925428390503, -1.243656873703003, 1.9822418689727783, 1.3206195831298828, 1.2688544988632202, -0.1219124123454094, 0.0859006717801094, 0.5940629839897156, -0.9194892644882202, 1.2586841583251953, 0.30530238151550293, -0.23717394471168518, 0.6773586273193359, 0.3372742533683777, 0.37593528628349304, 1.5013619661331177, -0.054069213569164276, 0.34607911109924316, 0.7163047194480896, -0.6907202005386353, 0.1178346499800682, -0.3512619435787201, -1.4574073553085327, -0.4479652941226959, -0.9102801084518433, -0.5405237674713135, -1.1264938116073608, -0.3107709586620331, -1.3411881923675537, -1.380548119544983, -0.32270845770835876, 0.5912142395973206, 0.32607612013816833, 0.4814590811729431, 0.6214360594749451, -0.9989975690841675, 0.20916059613227844, -0.6903374195098877, 0.216043621301651, 0.007573997601866722, -0.708039402961731, -1.0645639896392822, -0.40620267391204834, 0.9863999485969543, 0.6201285719871521, 0.6532416939735413, 1.839874267578125, -0.6347872018814087, -1.0868635177612305, 0.1985117793083191, 0.53235924243927, -0.7238705158233643, -0.025972584262490273, 1.1741459369659424, -1.110538125038147, 0.7925833463668823, -0.02829524502158165, 0.05781028792262077, -1.5496653318405151, -1.434819221496582, 0.15108722448349, -2.209876298904419, -0.8385778069496155, -0.40284454822540283, -0.5764366388320923, -1.1588057279586792, 1.0234335660934448, -1.0020310878753662, -0.8238552212715149, 0.5045208930969238, -0.1225544661283493, -0.7616977691650391, -0.3448377847671509, 0.27906665205955505, -0.4521181583404541, -0.14800214767456055, 0.7100792527198792, -0.6122466921806335, -0.5482983589172363, 0.33032456040382385, 1.0359665155410767, -1.3242557048797607, 0.763643741607666, -1.236634373664856, 1.973475456237793, -0.8504424691200256, 0.8550611138343811, 0.6126433610916138, 0.37573421001434326, -0.7999471426010132, 2.101046085357666, -0.5184791088104248, 1.2218458652496338, -2.254652976989746, -1.6423379182815552, -1.1237964630126953, -0.6973708271980286, 1.2715004682540894, -0.6430720090866089, 0.8472101092338562, -0.9758386015892029, -0.22409170866012573, -0.1593177765607834, -1.3968087434768677, 0.5571601986885071, 0.5328853130340576, -0.190664604306221, 0.35080355405807495, 0.207884281873703, 1.3945564031600952, -1.3456180095672607, 1.3206398487091064, -0.5977659225463867, 0.1909387856721878, -0.9710718989372253, 0.5108268857002258, -0.04179924353957176, 0.7720489501953125, -0.7151682376861572, -2.3936960697174072, -2.99747633934021, -0.18928685784339905, 0.15560033917427063, 1.4622904062271118, 1.0276793241500854, -1.9107918739318848, 1.0545367002487183, 0.7899931073188782, -0.16248434782028198, -0.004180399235337973, -0.5207382440567017, -0.13836669921875, -1.2858622074127197, -0.34420064091682434, -0.04283419996500015, 0.787748396396637, -1.5923579931259155, -0.46976712346076965, 2.14693284034729, 2.1651194095611572, 0.5942263007164001, -0.5338937044143677, 0.06549379229545593, 1.2904495000839233, -1.4291108846664429, 1.4310716390609741, 0.9161357879638672, 0.20659521222114563, 0.23954635858535767, -0.7553296089172363, 0.9830887317657471, -0.2918943166732788, -0.4724329710006714, -1.5675389766693115, -0.3944621980190277, 0.560996949672699, -0.5437209606170654, -0.5483051538467407, 0.19912095367908478, -0.05524180829524994, -0.6097046136856079, -0.031888097524642944, 1.7722409963607788, -0.27009132504463196, 0.989509642124176, -1.0689266920089722, 0.3761187195777893, 0.44216063618659973, -0.6730483174324036, -0.7825424075126648, -0.22079169750213623, -0.8231080770492554, 0.5889479517936707, -0.5590578317642212, -0.49919945001602173, 1.3109157085418701, 0.7182471752166748, -0.49478504061698914, -1.5825711488723755, 2.689669609069824, -2.0592572689056396, 1.5862195491790771, 0.6880909204483032, 0.8559226989746094, -0.8147873282432556, 0.304292768239975, -0.216425359249115, -0.48219233751296997, 2.3513097763061523, -0.4555555582046509, -0.6031431555747986, -0.06489034742116928, -1.783793330192566, 1.5723507404327393, -0.30781492590904236, 1.7286947965621948, 0.8461045026779175, -0.4838484823703766, 1.9868626594543457, 1.9998977184295654, -2.0164310932159424, -0.09255381673574448, -0.1445394903421402, -3.029628276824951, -0.45839789509773254, -0.6745506525039673, 1.0436983108520508, -0.7192987203598022, 0.6289197206497192, 0.586640477180481, -1.4704062938690186, 0.24711689352989197, 0.3164396286010742, -0.2907107472419739, -0.3571639358997345, 1.155350923538208, -0.5337336659431458, 0.48951616883277893, 1.8398927450180054, 0.3937780559062958, 0.8180463910102844, -1.6396158933639526, 1.3332691192626953, 1.7774648666381836, 1.3511935472488403, 2.151066541671753, -0.20103366672992706, 0.36035946011543274, -0.8810036778450012, 0.8754566311836243, -2.73088002204895, 1.212778925895691, 0.24374604225158691, -0.19617386162281036, 0.2903309464454651, 0.881193995475769, 0.10822939872741699, 1.1314276456832886, 0.30456748604774475, -1.277496099472046, 0.6001011729240417, -1.566109299659729, -0.2746567726135254, 0.9257941842079163, -1.005341649055481, 1.2432026863098145, 0.7668687701225281, 2.0827090740203857, 0.5295006036758423, -0.22072520852088928, -0.019363252446055412, -0.061318397521972656, 2.124955177307129, -0.2998834550380707, 0.2775587737560272, 2.2742726802825928, 1.2534083127975464, -0.5965477228164673, 1.2007339000701904, 2.0371134281158447, -0.1334388554096222, -1.416130781173706, -0.9973233938217163, -0.425182968378067, -0.06325937062501907, 0.7216413021087646, 1.5337316989898682, 0.47342392802238464, -0.6600456237792969, 1.024320363998413, -1.6966418027877808, 1.2249399423599243, -0.48022645711898804, -0.8949686288833618, -0.16021360456943512, -0.1796875149011612, -1.08824622631073, 0.9206390976905823, 0.026069574058055878, 0.32677698135375977, 1.0665613412857056, 1.2701869010925293, -0.0945703312754631, -0.4822969436645508, -1.0254863500595093, -0.9415372610092163, -1.488411545753479, -2.2300281524658203, 0.6451090574264526, 0.20102733373641968, -1.6321935653686523, 1.3223131895065308, -0.2000451534986496, 0.7520303130149841, -0.2607591450214386, 0.31458255648612976, -1.6756479740142822, -1.9061648845672607, 0.2002040594816208, 0.4385163187980652, -1.9517103433609009, -0.07696215063333511, -0.18808972835540771, -0.43920737504959106, -2.016942024230957, -0.3392205536365509, 0.9263833165168762, 0.3385806679725647, -0.7831880450248718, -0.2553566098213196, -0.5869652032852173, 0.18396545946598053, 0.8352781534194946, 0.018374156206846237, 0.17926998436450958, -2.391524076461792, 0.19791485369205475, -1.0253838300704956, -0.2632621228694916, 1.3373541831970215, 0.4843839406967163, 0.23131786286830902, 0.5257962346076965, 0.919921875, -1.8796244859695435, 0.43588629364967346, -0.38495132327079773, 0.7766459584236145, 0.04429326206445694, -0.2641758620738983, 0.30607670545578003, 0.30524682998657227, -0.8617695569992065, -0.537646472454071, -0.8043726682662964, 0.7482813000679016, 0.31535768508911133, 0.01891849748790264, 0.7423099875450134, 0.34998613595962524, -0.9535002708435059, -0.6948729753494263, -0.38095715641975403, -1.204318642616272, -1.2275362014770508, -0.9143527150154114, 0.9557857513427734, -2.8044745922088623, 1.059561848640442, 0.7659628987312317, -0.7889937162399292, -0.7014293670654297, 0.1894373595714569, -1.8689079284667969, -1.6935168504714966, 1.5691345930099487, -2.540717601776123, -0.36335158348083496, 0.4657207727432251, -1.9456192255020142, -0.7478064298629761, 0.3551901578903198, -0.3701687455177307, -2.6983041763305664, -1.339138150215149, -1.170738935470581, -1.2285101413726807, -1.0214006900787354, -0.19654087722301483, -0.9771827459335327, 0.19525517523288727, -0.18085841834545135, -1.5570586919784546, -1.6840943098068237, 0.22192007303237915, -0.8085981011390686, 1.0699838399887085, 0.02580607496201992, 1.8913909196853638, -0.19341106712818146, 1.459442138671875, 2.0645720958709717, -1.3605592250823975, 1.133182406425476, 0.002228576224297285, -0.77811598777771, -0.7105036377906799, 1.821548342704773, -0.8883728384971619, 0.553229033946991, 1.7016223669052124, 1.3293976783752441, 0.2895016074180603, -0.7808266878128052, 0.2625146508216858, 0.7795015573501587, 0.6432945132255554, -1.2700800895690918, -0.1537197083234787, -1.0585969686508179, -0.49574723839759827, 0.07399678230285645, -0.3077177107334137, 2.3465654850006104, 0.457387238740921, 0.18738123774528503, -0.23124128580093384, -1.5072426795959473, -1.724632978439331, -1.616122841835022, -0.1434279978275299, -1.0718660354614258, 0.5509098768234253, 0.5135353207588196, -1.1680964231491089, -0.6487354636192322, -1.1629384756088257, 2.4116594791412354, -0.6327985525131226, 0.5999398827552795, -0.8732293844223022, -0.40431833267211914, -0.3615812361240387, 0.4806012213230133, 0.7406958341598511, 1.1221181154251099, 0.49676305055618286, 0.24273718893527985, 0.3277832567691803, -0.32765525579452515, -1.0126961469650269, -1.065120816230774, 0.5319446921348572, -0.6993522644042969, -0.8820978999137878, -0.45619747042655945, 1.648146629333496, 0.6628739833831787, -1.259537696838379, -0.666980504989624, 1.216007113456726, 0.3976397216320038, 0.12520189583301544, -0.23556402325630188, -0.17686063051223755, 0.8646597266197205, 1.041053295135498, 0.0228981152176857, -0.1791282743215561, -0.8182570338249207, -0.17652550339698792, 1.7823371887207031, -1.7147661447525024, 2.5189549922943115, -0.8339953422546387, -0.5663238167762756, 0.12829473614692688, 0.43378132581710815, -0.2458803653717041, 1.3054375648498535, 0.031696218997240067, -0.26404303312301636, -0.2940470278263092, -0.5035943984985352, 0.42992082238197327, 1.111629843711853, 0.24162140488624573, -0.4207863509654999, -0.045740630477666855, -0.18333996832370758, 0.48664942383766174, -2.236053943634033, 0.6938508749008179, -1.2846832275390625, -0.7462388277053833, 1.7012743949890137, -0.5525158643722534, -0.9319689273834229, 0.3787408471107483, -2.523603916168213, -0.41829660534858704, 0.4126921594142914, 0.1823558360338211, 0.16983574628829956, 0.9903432130813599, 0.24410203099250793, -0.23362098634243011, 1.4323679208755493, 1.6635642051696777, 0.5060612559318542, -0.9172171950340271, -0.9600425362586975, -0.14159275591373444, -0.7214581966400146, 1.3817055225372314, -1.5861482620239258, 1.1760977506637573, 0.7344164848327637, 2.8096861839294434, -0.18887795507907867, -1.0829850435256958, 2.180922031402588, -0.0021792841143906116, -0.8688264489173889, 0.9025075435638428, 1.0197457075119019, 0.8383838534355164, -0.12345607578754425, -0.28571438789367676, -0.328753262758255, 1.0493632555007935, -0.3684503734111786, 1.5388530492782593, 0.6715688705444336, -0.5445414781570435, 0.8832524418830872, 0.05792972445487976, 1.018410086631775, 0.6648023724555969, -0.03514397516846657, -0.0945451483130455, -1.5425103902816772, -0.3786863386631012, -0.5047151446342468, 1.3128671646118164, -2.088677406311035, -0.671596348285675, 0.44160038232803345, 0.2689138948917389, -0.20843228697776794, -0.23273378610610962, 1.9742655754089355, 0.12149164080619812, -0.8304160833358765, -1.2427806854248047, 2.465134859085083, -1.8265658617019653, 1.301804542541504, 1.064613938331604, 2.242076873779297, -2.0218875408172607, -0.3825095295906067, 0.26638540625572205, 0.4549282193183899, 0.42957645654678345, -0.31019067764282227, 0.2764928936958313, -0.20679455995559692, 1.4208661317825317, 0.6909312009811401, -1.0296310186386108, 1.5628652572631836, -0.30873167514801025, 0.056123968213796616, 0.7111945748329163, -0.0512070395052433, -0.08902047574520111, 0.7753908038139343, -0.4636997878551483, 0.7008413672447205, -0.011007637716829777, 0.16763073205947876, 1.0622254610061646, -0.2052956372499466, 1.6894029378890991, 1.3334949016571045, -0.8946167230606079, -0.2825816869735718, -2.213402271270752, 0.8420047163963318, 1.6509504318237305, -0.7171887159347534, -1.6655170917510986, -1.2814922332763672, -0.055480439215898514, -0.7296499013900757, 0.610926628112793, 0.996768593788147, 0.2700483202934265, 0.390935480594635, 0.8003931045532227, 1.1123604774475098, 0.4007073938846588, -0.3855077028274536, 2.042457103729248, 0.05773327872157097, -0.15234674513339996, -0.5512702465057373, -1.0176007747650146, 1.0833014249801636, 0.3146946430206299, 2.2822024822235107, -0.5156490802764893, -1.0502444505691528, 1.8926987648010254, 0.0712222009897232, -0.5766291618347168, -0.5318116545677185, 0.9054747223854065, 2.2743167877197266, 1.1252110004425049, 0.26544836163520813, 0.9274382591247559, 1.0421606302261353, -2.259707450866699, 1.4707180261611938, -0.8699131608009338, 0.03572291135787964, 0.8687359690666199, -0.5448545217514038, -0.48157331347465515, -1.3364813327789307, -0.2273375391960144, -1.3675410747528076, -1.4057332277297974, -0.24111296236515045, -0.5574018955230713, -1.250001072883606, 1.5128364562988281, 0.5655166506767273, 1.6041955947875977, -0.26536720991134644, -1.6177887916564941, -0.564367949962616, 0.3704708516597748, 1.3694067001342773, 0.30116355419158936, -1.8485374450683594, 0.1645943820476532, -0.9150019288063049, 0.013231057673692703, 1.26040518283844, 2.600743055343628, -1.509254813194275, -0.009105455130338669, 2.555860996246338, -0.09210033714771271, -0.31227973103523254, 1.729872465133667, 0.9597252607345581, -0.2584671676158905, -0.5067646503448486, 0.4607452154159546, -0.9647778272628784, -0.948163628578186, 0.3846905529499054, -1.6834744215011597, -0.13975882530212402, 0.6034355759620667, -0.3942343294620514, -0.0015588661190122366, -1.483262300491333, 0.2999204993247986, -0.8669785261154175, 0.4163249731063843, 0.17029839754104614, 1.0250332355499268, -1.1264718770980835, 2.0527119636535645, -0.26320308446884155, 0.5131165981292725, 0.0537789985537529, 1.1356136798858643, 0.09497451782226562, -1.2799330949783325, -0.18325552344322205, -0.00327819911763072, -0.3977276682853699, -0.9300636649131775, 0.7873576283454895, 1.7516413927078247, -0.2435319721698761, 0.8960981369018555, 0.2977263331413269, 1.5234334468841553, 0.42848679423332214, 1.7134569883346558, -2.3826677799224854, 1.7854236364364624, 0.3365369737148285, 0.1002330407500267, 0.2710722088813782, -0.881833016872406, 1.040026307106018, 0.6369179487228394, -0.5298845767974854, -0.44784846901893616, -1.63343346118927, 0.10765790194272995, 1.525132179260254, 0.5625307559967041, -0.031053345650434494, -1.1757498979568481, -0.23804213106632233, 0.5092217326164246, -1.545758605003357, -0.6423436999320984, -0.10857133567333221, -2.3139808177948, -0.9212362766265869, 0.09406402707099915, 1.9887404441833496, -0.1082635372877121, -0.5169367790222168, 0.5215975642204285, 0.3430671989917755, -0.06956781446933746, 0.4180079698562622, 0.3612603545188904, -0.575650155544281, -0.14959204196929932, -0.26400089263916016, -1.0406194925308228, -0.7042030692100525, 0.9991884827613831, -1.41741943359375, -1.0454535484313965, 0.8684406876564026, 0.38708677887916565, -0.0374787300825119, 0.9371348023414612, 0.39468806982040405, 0.14654503762722015, -0.9260170459747314, -0.33294978737831116, 0.3318611681461334, 1.2272182703018188, 0.5615628361701965, 2.0960705280303955, -0.34051597118377686, 0.8678765296936035, -1.299674391746521, 0.34190651774406433, 0.5374740362167358, 0.18468986451625824, -1.7571520805358887, 0.3379687964916229, 1.7977582216262817, -0.7996687889099121, 0.058945029973983765, -0.2625954747200012, -0.4062515199184418, 0.13194073736667633, -0.6103842854499817, -0.6004093289375305, -0.01709756627678871, -1.0642528533935547, -0.21341854333877563, -0.3200168311595917, -0.2764284908771515, 1.4362616539001465, 0.7407044768333435, 0.3057151138782501, -0.5243791937828064, -1.140363335609436, -0.9403443336486816, 0.5534172654151917, -0.021822744980454445, -0.3539135158061981, 2.1665210723876953, -0.178741455078125, -0.12250475585460663, -2.3127987384796143, -1.379750370979309, -1.3086329698562622, -1.4497488737106323, -1.4303069114685059, -1.3892525434494019, 1.5351213216781616, -0.5113720297813416, -0.027553625404834747, -1.5710731744766235, -1.2121509313583374, -0.031306155025959015, 1.3813966512680054, 1.2312151193618774, -0.3347422182559967, -0.5009159445762634, -0.4393598139286041, -0.7250067591667175, -0.06964600831270218, -0.8306040167808533, 0.22149866819381714, -0.2413136512041092, 0.5315171480178833, -0.6909133195877075, -0.643300473690033, -0.8687741756439209, 0.06505343317985535, -1.1129878759384155, -0.5625895261764526, -0.16392965614795685, 0.4158095717430115, -0.441234290599823, -0.18027135729789734, 1.4530341625213623, 0.4526340663433075, -0.9905833005905151, 0.124837726354599, 0.1445431113243103, 0.0771603062748909, -0.8847367763519287, -2.637125015258789, 1.5954800844192505, 0.2746604382991791, -0.14881457388401031, 0.48068544268608093, -0.5745813846588135, -0.3853274881839752, -0.21427962183952332, 0.1323719173669815, 2.049837827682495, -0.7626364827156067, 0.7262648940086365, 0.04981725662946701, 0.5445912480354309, -0.7301279306411743, -0.9110395312309265, 0.8983495235443115, -0.06867951154708862, -0.5513642430305481, -0.14672735333442688, -0.22094039618968964, 0.07891591638326645, -0.1451430767774582, 0.19838158786296844, -1.4412364959716797, -0.7014840245246887, 1.4904375076293945, -0.9696798920631409, 2.5240185260772705, -0.21302224695682526, 0.06354989111423492, 0.7216174006462097, 0.05298170447349548, -2.1328365802764893, -0.1435830444097519, 3.2228307723999023, 0.022250449284911156, -0.14600186049938202, -0.3225328326225281, -0.8367542624473572, 0.4393733739852905, 1.2035233974456787, -2.0392606258392334, 1.1250120401382446, 0.3074333369731903, -0.5875252485275269, 0.6354179978370667, -1.07755446434021, 0.6922176480293274, 0.9042142629623413, 1.1255978345870972, 1.5800327062606812, -1.4862552881240845, -0.8485028743743896, -0.4500770568847656, -0.267343670129776, -1.1601661443710327, -1.483150839805603, 0.6908363103866577, -1.0996816158294678, -0.5065066814422607, -1.5061594247817993, 1.2813912630081177, 0.12073855847120285, -1.6128069162368774, 0.484988808631897, -0.8721350431442261, -0.06762643903493881, 1.0691242218017578, -1.2084813117980957, -0.41048163175582886, 0.5715974569320679, -0.2596801817417145, 1.0644276142120361, 0.4107220470905304, 0.9305852055549622, -0.40052273869514465, 1.1919621229171753, 0.5307913422584534, 1.3320928812026978, -2.145181179046631, 0.0985843762755394, 1.4044396877288818, -0.28195616602897644, 0.42865630984306335, -0.1568794548511505, -0.22814778983592987, -0.0930127501487732, -0.263899564743042, -0.5868462324142456, 0.5897418856620789, -0.17117846012115479, 0.44098466634750366, 0.7770484089851379, 0.9973829388618469, 0.12236914783716202, 0.966885507106781, 0.053645167499780655, 0.5047204494476318, 1.146228313446045, -0.9734684228897095, -1.2050131559371948, 0.20861537754535675, -1.8427197933197021, -1.6525603532791138, 0.4635477662086487, 1.147955060005188, 0.18292702734470367, -0.8703806400299072, 0.3352043926715851, -0.3078053295612335, 0.4963054060935974, 1.3389114141464233, -0.9643958210945129, 0.8495247960090637, -1.250089406967163, 1.3181123733520508, -0.2986326515674591, 1.8849273920059204, 0.8319960236549377, 0.08599379658699036, 0.40479207038879395, -0.07357363402843475, 0.8163150548934937, -0.32922640442848206, -0.8985200524330139, -1.1366550922393799, -1.1334495544433594, -0.3977639079093933, 1.8532416820526123, -0.1912107914686203, -1.486073613166809, -0.24877706170082092, 1.003952980041504, -1.053836703300476, 1.4856398105621338, 0.63984614610672], [1.692150592803955, 0.1455085426568985, -0.2751963138580322, -1.2294375896453857, 0.14218348264694214, 0.2763040363788605, 0.3828781545162201, -1.3210822343826294, -0.39084842801094055, 0.794998288154602, -1.6298178434371948, 0.6020826697349548, 0.24054837226867676, 0.9897816181182861, -0.07351750135421753, -1.026234745979309, 0.1901632696390152, 0.13897104561328888, 1.4816910028457642, 1.5605578422546387, -1.6581947803497314, 0.738963782787323, 1.0387027263641357, 1.8680176734924316, -0.6486099362373352, -0.6706759929656982, 0.5288040041923523, -0.19695395231246948, 0.33018186688423157, 1.1846994161605835, -1.3384907245635986, 1.0111827850341797, -0.1218714788556099, 0.18122634291648865, 0.07716268301010132, -1.645278811454773, -0.022354615852236748, -1.2088781595230103, 0.7397778630256653, 0.14840316772460938, -0.8767499327659607, -0.8017566204071045, 1.84714674949646, 0.7362485527992249, 0.8544262051582336, 1.143115758895874, 0.43269768357276917, 0.12306051701307297, 1.2814205884933472, 2.1281003952026367, 1.6245181560516357, 0.5041548609733582, 0.42733651399612427, 1.2608953714370728, -0.3393281400203705, 1.1398805379867554, 0.28681886196136475, 1.4031401872634888, -0.26192528009414673, -1.3219084739685059, -0.4233814477920532, 0.2856099605560303, -1.5901713371276855, 0.723203718662262, -1.8886792659759521, 0.11338864266872406, -0.511425256729126, -0.37998461723327637, 1.2222343683242798, 0.4798472821712494, -0.30107054114341736, 1.6997398138046265, -0.6167089343070984, -1.158575415611267, 0.1552184373140335, -1.2841918468475342, 0.2215091586112976, 0.12313953042030334, -0.9367501139640808, 0.6012459397315979, -1.1140598058700562, 0.6843182444572449, -0.18913879990577698, 0.09089729934930801, -1.0647950172424316, 0.1267474889755249, -0.16033291816711426, 1.052901268005371, -0.45860809087753296, 0.5514456033706665, -0.8129850625991821, 0.386393666267395, 0.7184940576553345, 0.8751330971717834, 0.9353840947151184, 1.7257978916168213, -0.897193431854248, 0.5903767347335815, 0.2724817097187042, 0.1585955172777176, 0.1475498378276825, 0.504677414894104, 1.5881695747375488, 0.15599225461483002, 1.4195462465286255, 0.6676110625267029, 0.6060754060745239, -0.00959706399589777, 0.013849538750946522, -0.5669296383857727, 1.1383053064346313, -0.10053399205207825, -0.5455878973007202, 0.3169094920158386, 0.46414265036582947, -0.9698793888092041, -1.9293550252914429, -0.30204835534095764, 0.31841182708740234, -0.10264316946268082, 1.6696536540985107, -0.9202149510383606, 0.24387599527835846, 0.09221271425485611, 0.40142813324928284, 0.3799731433391571, -0.28841742873191833, -0.7709118723869324, 0.16848629713058472, 0.19453178346157074, 1.0312119722366333, 1.2508797645568848, -0.4522710144519806, -1.6172856092453003, -0.40503251552581787, -1.064738154411316, 0.333351731300354, -0.18142719566822052, -0.13813868165016174, -0.03564954176545143, 0.16660279035568237, 0.07273533940315247, -1.3823294639587402, 0.5699157118797302, 1.435361623764038, 0.5151181817054749, -0.6357861757278442, -1.2528109550476074, 1.1346004009246826, -2.5287559032440186, -0.5534977912902832, 0.35063502192497253, 0.7578007578849792, 0.15637801587581635, -1.8186488151550293, -0.728618323802948, -0.9473675489425659, -0.10904520750045776, 0.582183301448822, 1.9903486967086792, -2.1521177291870117, 1.6032973527908325, -1.4933890104293823, 0.03358510881662369, -1.23041832447052, -1.5114191770553589, 0.6073672771453857, 0.6807593107223511, -0.3499971926212311, -1.3043649196624756, 2.2261171340942383, -1.7031868696212769, 0.1740068942308426, -1.0685571432113647, 0.03579917922616005, 0.6905941367149353, 1.0393952131271362, 0.5472099781036377, 1.2520719766616821, -1.817554235458374, 0.9586018323898315, 1.5066299438476562, -1.5038869380950928, 1.0438584089279175, 0.400208055973053, 0.6045823097229004, -0.15795400738716125, 1.1109124422073364, 1.2173354625701904, 0.5098971724510193, -1.5838733911514282, -1.324237585067749, 0.6552616357803345, -1.1079174280166626, -0.32587486505508423, 0.7621170282363892, 0.5425750613212585, -1.7879189252853394, -0.005979010369628668, -0.12387330830097198, 0.01845467835664749, -0.6990386843681335, -1.9370824098587036, 1.5874254703521729, 0.9783374667167664, 0.4035186767578125, 0.4775177538394928, -0.04356573894619942, -0.5732792019844055, -0.8485749363899231, 1.9288709163665771, -0.21179616451263428, -0.00707818940281868, -1.1105868816375732, -0.2603817582130432, -0.6321570873260498, -1.1305333375930786, -2.2825400829315186, 0.8927964568138123, -0.027708610519766808, -0.892687201499939, 2.6959338188171387, -0.09159950911998749, -0.6235485672950745, -0.7391102910041809, -0.919649600982666, 0.27880215644836426, 2.3320963382720947, 0.2294372022151947, 1.1174131631851196, -0.44009506702423096, -0.9813344478607178, -0.3457046449184418, 0.07252753525972366, 2.528691291809082, -0.17051571607589722, 0.37293124198913574, -0.8174551725387573, -1.2575623989105225, 0.016682922840118408, 0.7757322192192078, -0.05806005746126175, -0.5026000142097473, -0.28349146246910095, 0.7182590365409851, -0.4106046259403229, 0.8508830070495605, -0.9271066188812256, -0.1083800345659256, 0.39709940552711487, 1.3427913188934326, -1.780273675918579, 0.2245386689901352, -1.3066664934158325, -0.0355900414288044, 0.5149999260902405, -0.5552293658256531, -2.428842544555664, -0.03718012943863869, 0.29934269189834595, 0.20795764029026031, 0.4971272945404053, 1.2696443796157837, -0.37192127108573914, 0.42932114005088806, -1.3926293849945068, -2.3464863300323486, -0.3251470923423767, 0.08593778312206268, -1.3078641891479492, 0.6101229190826416, 0.8217878937721252, -0.5987414717674255, 1.8620349168777466, -0.9827601313591003, -0.8127509951591492, 1.3531454801559448, 0.8642476201057434, 0.8561396598815918, -1.797687292098999, 0.5051900744438171, 1.7014225721359253, -1.4199079275131226, 0.20340442657470703, 0.385780394077301, 0.35483014583587646, -2.083042860031128, -0.8213672637939453, -1.7004785537719727, 0.043787579983472824, 0.1666235327720642, -0.7224851846694946, 0.6780899167060852, 0.6046584248542786, 0.8677030205726624, -0.732483446598053, 2.0704312324523926, 1.8418203592300415, -1.0736420154571533, 0.4264073371887207, -0.7140084505081177, -0.016699258238077164, -0.5946222543716431, 0.5205696821212769, 0.17709901928901672, -0.39223313331604004, -0.12333789467811584, -1.1354179382324219, 0.10027153044939041, 0.2056807428598404, 1.017688274383545, 1.1350784301757812, -0.061384446918964386, -0.09628963470458984, 1.211796760559082, 1.2639106512069702, -0.6894667744636536, 0.4628114402294159, -0.3348904848098755, -1.5639798641204834, 0.34462234377861023, -1.2513636350631714, -1.7221224308013916, 1.891410231590271, 1.5933418273925781, -1.3947100639343262, 0.8857285976409912, 1.1761809587478638, -0.9527139663696289, -1.0065529346466064, -0.31408369541168213, 0.5753118991851807, 0.7126309275627136, -1.0937715768814087, -0.40817341208457947, -1.6961300373077393, 1.217255711555481, -1.5360246896743774, -1.1373651027679443, -0.4712994396686554, -0.052492305636405945, 0.7083286046981812, 1.2367558479309082, -0.5392897725105286, -0.16318169236183167, 0.8363322615623474, 0.5709284543991089, -1.2122691869735718, -0.6997572779655457, 0.4821176826953888, -0.9799597859382629, -0.6652068495750427, -1.2043755054473877, 0.7760798335075378, 0.10112403333187103, -1.1594451665878296, -0.382655531167984, 0.1858747899532318, 0.35195687413215637, -0.4766966998577118, 0.9644774198532104, -0.9803864359855652, 0.08044959604740143, 1.0777305364608765, 1.8055732250213623, 0.9794662594795227, 2.4533443450927734, -0.34807538986206055, -1.9155346155166626, -0.7965982556343079, 0.5537793636322021, 0.9846179485321045, -1.2506779432296753, 0.22449786961078644, 0.49974262714385986, 0.44938498735427856, -1.2242854833602905, -0.558712363243103, 0.1595253348350525, 0.7946129441261292, 1.5329192876815796, 1.4021669626235962, 1.159155249595642, 1.2391961812973022, -0.31364747881889343, 0.5771666169166565, -2.0655696392059326, -0.6260474920272827, -0.6371886730194092, 1.1131511926651, -1.3811194896697998, -0.5400198101997375, 2.807032823562622, -0.5594826340675354, -0.4305769205093384, -2.3425543308258057, 0.21656732261180878, -1.1845667362213135, -0.2133457511663437, -1.0991779565811157, 1.3142929077148438, 1.1770024299621582, -1.1334834098815918, 0.018427008762955666, -0.8276455402374268, 1.015840768814087, -0.6267693638801575, 0.9052765369415283, 0.2853946387767792, 0.08245360106229782, -1.2513158321380615, -1.1135196685791016, 1.4119621515274048, 0.07457006722688675, 0.17576946318149567, 1.6365517377853394, 0.5059093832969666, 1.4935895204544067, -0.13385161757469177, 0.6826701760292053, -0.3419572412967682, 0.6853048205375671, -0.9988660216331482, 1.0687718391418457, 0.5041431188583374, -0.3037673532962799, 1.7525012493133545, 0.538381814956665, 0.8942413926124573, 2.2058920860290527, -0.5256382822990417, 0.2920148968696594, 0.692939043045044, 0.33309993147850037, 0.4667281210422516, -0.22464172542095184, -1.3630813360214233, 0.20531073212623596, -0.5408884286880493, 1.3182872533798218, -1.0735063552856445, -1.4855947494506836, 2.0439651012420654, 0.3237271010875702, 1.7917143106460571, -1.071834683418274, 0.8091291189193726, -0.5740086436271667, -0.7290400266647339, -1.16376793384552, 2.478614330291748, 0.47050750255584717, 0.4802645146846771, -0.4668098986148834, -0.9883017539978027, -0.7708352208137512, -1.0080320835113525, 0.3225077986717224, -1.0805954933166504, -1.882384181022644, 0.3916584551334381, 1.2076447010040283, 2.16770601272583, -1.126693844795227, 0.17685478925704956, 0.0005557708791457117, 0.3675781190395355, -1.4317243099212646, 1.1091139316558838, 1.253998041152954, -2.2000980377197266, 0.8865570425987244, -0.8722085952758789, 0.18914753198623657, 0.1034754291176796, 0.025991354137659073, -0.5509837865829468, 0.3000949025154114, 0.09019789844751358, 0.6195142269134521, 0.1714259833097458, 0.365813285112381, -0.08555790781974792, 0.41679152846336365, -1.6527613401412964, -0.7795416116714478, -0.6352086067199707, 0.13076266646385193, 0.9511889815330505, -1.9023200273513794, -0.47054389119148254, -0.0961129367351532, -0.21553096175193787, 0.07368610054254532, -0.827539324760437, 0.6511892676353455, 0.07684452086687088, -1.0405935049057007, -1.3509563207626343, -0.7966288924217224, -0.07281960546970367, -1.7119429111480713, 1.4340966939926147, -0.22067342698574066, -1.0476889610290527, -0.6542132496833801, -1.2699337005615234, 0.11449309438467026, -1.0631307363510132, -0.11795783787965775, 0.28111180663108826, -0.06204286962747574, -0.10547909140586853, -0.048494089394807816, 0.21589583158493042, 0.017720825970172882, -1.1522035598754883, 1.2654281854629517, 1.520438551902771, -0.19304773211479187, -0.9524698257446289, -0.924714207649231, 0.4866708219051361, 0.6857408285140991, -0.10634582489728928, 0.9565490484237671, 1.4546778202056885, -1.0311286449432373, -0.24077282845973969, -1.2187434434890747, 0.10512992739677429, 0.07931606471538544, 0.41719210147857666, 1.4107288122177124, 0.6263439655303955, -0.5540790557861328, 1.803521990776062, -1.6199311017990112, 0.021966107189655304, 0.012318446300923824, -0.5080002546310425, 1.070105791091919, -2.0115339756011963, 1.1751267910003662, 0.048966359347105026, -2.1786577701568604, 0.3709651231765747, 1.701174259185791, -0.2752537131309509, 2.1541473865509033, 1.4927401542663574, 1.3317160606384277, -1.361161231994629, -0.018356533721089363, -0.2891696095466614, 1.3084150552749634, -0.1991892009973526, -2.032438039779663, 0.4204120635986328, 0.6990043520927429, 1.21867835521698, -0.19573500752449036, -1.7960233688354492, -0.7412578463554382, -0.38144174218177795, 0.17551057040691376, 0.6582090854644775, -1.3378690481185913, 0.004516397602856159, -0.5066810846328735, -0.2630043625831604, 0.5080447793006897, -1.7155240774154663, -0.8031196594238281, 0.367637574672699, -0.6616647243499756, 2.0608556270599365, -0.8955504298210144, -2.2847673892974854, 0.2633216679096222, 0.4790842831134796, -0.23997235298156738, 0.586459755897522, -0.9399822354316711, -0.6453793048858643, -0.24225112795829773, 0.8282812237739563, 0.31275057792663574, 1.101143479347229, -0.049232471734285355, -0.6983633637428284, -0.7967407703399658, 0.7905147075653076, 0.38879314064979553, 0.262407511472702, -0.22262655198574066, 1.1234500408172607, 0.11287561058998108, 2.0719521045684814, 1.270954966545105, 0.2352457493543625, 0.13077762722969055, -0.7004339694976807, 0.23135431110858917, 0.18005840480327606, 0.707952082157135, 1.063001275062561, -0.4273685812950134, -0.7504131197929382, 1.0202159881591797, -0.878967821598053, -0.991949737071991, -0.4974682033061981, 0.007479456719011068, -1.5557245016098022, 0.5235667824745178, 1.753390908241272, -0.4638206660747528, -0.3651185631752014, 0.0432477667927742, -0.5656521916389465, 0.14287368953227997, 0.1933310627937317, 1.026904582977295, 0.5427876114845276, 1.3685482740402222, -0.5673483610153198, -0.0941273421049118, -0.7833535671234131, 0.2679755389690399, 0.10544434934854507, 1.4683974981307983, 2.1121280193328857, -1.9787683486938477, -2.461529016494751, 0.47451135516166687, 1.4333621263504028, -0.7763545513153076, 1.7318358421325684, -0.8589493632316589, -0.6010810732841492, 0.553636908531189, -1.4715334177017212, 0.43524953722953796, 0.02381824143230915, -0.10113371163606644, 0.20467524230480194, -0.43982163071632385, 0.05845864862203598, 1.7129489183425903, -0.4532485604286194, -1.8899744749069214, 1.648890733718872, -0.24762386083602905, -0.2357844114303589, -1.6049808263778687, 0.310888409614563, 0.319070965051651, 0.6620402932167053, -0.197770357131958, 0.06674405932426453, 0.6064227819442749, -0.5463379621505737, -0.10479771345853806, -0.6458830833435059, 0.2008494734764099, -0.6864643692970276, 0.7292765378952026, 0.8415611386299133, -0.38898950815200806, -0.19091321527957916, -0.266705721616745, 1.485245704650879, -0.9549651741981506, -0.8992392420768738, 0.5381684303283691, -0.48980310559272766, -0.8506244421005249, 0.7914997339248657, 1.0046478509902954, -1.1312693357467651, 0.019668519496917725, 0.7388855814933777, -0.9469059705734253, 0.9402849674224854, 2.0102474689483643, 1.0379509925842285, 0.7788377404212952, 0.7931930422782898, -0.5985197424888611, 1.7572795152664185, 1.361282467842102, -1.718214511871338, 0.3038824498653412, 1.4088670015335083, 0.6748633980751038, 1.1149965524673462, 0.5808597207069397, -0.9091092348098755, -0.26736000180244446, -0.6737182140350342, 0.6043368577957153, -0.3892219364643097, -0.9238999485969543, 0.3387311100959778, 1.0627082586288452, -0.28250911831855774, -2.158689260482788, 0.5343328714370728, -2.3870487213134766, 0.3890618085861206, 1.5135942697525024, -9.45705542108044e-05, -0.9163665175437927, -1.3090410232543945, -0.654625654220581, 0.6610221862792969, -1.975428581237793, 0.610368549823761, -0.18715929985046387, 0.7585330605506897, 1.019736409187317, -0.10904958844184875, -0.750821053981781, 0.397255539894104, -0.7851778268814087, 0.2656676769256592, -0.514945387840271, -1.3183884620666504, 0.26524561643600464, 0.6251658797264099, -0.2016574889421463, -2.329983711242676, -0.26670292019844055, -0.30366310477256775, 0.8382403254508972, 1.3124585151672363, -0.801766037940979, -0.14945249259471893, -0.9541590809822083, 2.4085116386413574, -0.7716753482818604, -1.3527030944824219, -0.08229149878025055, -0.4415530264377594, -0.07206887751817703, 0.3676963746547699, 1.094277024269104, 0.26555800437927246, 1.3343762159347534, 0.9696140289306641, 0.548652708530426, 0.5013306140899658, 1.3813918828964233, -0.11884958297014236, 0.9240701794624329, -0.4128634035587311, 1.3720544576644897, 0.4160679578781128, -1.6038044691085815, -0.27825793623924255, -0.40275418758392334, 2.382538080215454, 1.7405054569244385, -0.2967014014720917, -0.1399124264717102, -0.4759918749332428, -0.10705027729272842, -1.6244958639144897, 1.815344214439392, 0.11605563759803772, 0.089653879404068, 0.5383006930351257, 1.5361456871032715, 0.22056792676448822, -0.08805634826421738, 1.4623678922653198, -0.029763681814074516, -0.20595470070838928, -0.5959097146987915, 0.5380665063858032, -0.5463088750839233, -0.8256188631057739, 0.6011127233505249, -2.25388503074646, -0.21691106259822845, -0.5401690602302551, -1.649277687072754, 2.189345359802246, -0.3373963236808777, -1.5539900064468384, -0.030966755002737045, -0.2409043312072754, 0.5781055092811584, 1.2902238368988037, -1.1158106327056885, 0.2217576652765274, -0.2542375326156616, -4.941127190249972e-05, 1.2245551347732544, -0.4539026916027069, -1.1628879308700562, 0.4815524220466614, -1.0266826152801514, -0.28937575221061707, 1.0688568353652954, -0.5894790291786194, -0.9196400046348572, -0.05291550233960152, 0.9299290776252747, -1.2678133249282837, -0.08331404626369476, -1.9434128999710083, 0.31185466051101685, 0.35140204429626465, -0.4030874967575073, 1.3941055536270142, 0.13953113555908203, -1.6599006652832031, 0.9483810663223267, 0.7791368365287781, 0.6484998464584351, 1.510462760925293, 0.26726263761520386, 1.1645922660827637, -1.1317486763000488, 0.9885251522064209, 0.2018909454345703, 0.34874778985977173, -0.9671011567115784, -1.1442420482635498, -0.6125802993774414, 1.7573542594909668, 0.8188132643699646, 0.6731106638908386, 0.8156347274780273, 0.00021774017659481615, 0.5519331097602844, -1.2968453168869019, -0.9922578930854797, -1.7973942756652832, -0.7355697154998779, -2.1762049198150635, -0.20047618448734283, -0.18829768896102905, 2.3187813758850098, 0.8111242055892944, 1.8154233694076538, 0.32227930426597595, -0.4325455129146576, 1.1225601434707642, -0.5557959079742432, -0.5380494594573975, 0.940523087978363, -1.2416496276855469, 0.5195271968841553, 0.415340781211853, 1.3893930912017822, -0.6885547637939453, 0.17116685211658478, -0.154921755194664, -1.5953844785690308, -1.634934663772583, 0.35989996790885925, 0.5486517548561096, 2.2307357788085938, -0.5734147429466248, 0.5176881551742554, -1.3818780183792114, -1.5288246870040894, 0.37638556957244873, 1.6825203895568848, -0.19818885624408722, 0.40574464201927185, 0.8395689725875854, 0.706824541091919, -0.3593479096889496, 0.09140467643737793, 1.9529753923416138, 0.25875529646873474, 0.2062326818704605, 1.2314701080322266, 0.35800743103027344, -1.163496732711792, -1.731850266456604, -0.3109799921512604, 0.8726240396499634, 0.5935463905334473, -0.2155570089817047, 0.7365021705627441, -0.17588789761066437, 0.6813037395477295, -1.1541969776153564, 0.011945899575948715, 1.00250244140625, -0.64171302318573, 0.8610947132110596, 0.05513441935181618, 0.2092466950416565, -0.24543829262256622, 1.5528525114059448, -0.013266794383525848, -1.0331051349639893, -0.8387880921363831, -0.7353631854057312, 0.7581368684768677, -0.40408721566200256, 0.5854170322418213, -1.8195788860321045, -0.4398562014102936, -0.7828829288482666, -1.628941297531128, -1.2627040147781372, 0.4467492997646332, 0.0461602509021759, -0.06070338189601898, 0.3222516179084778, -1.1704316139221191, -2.0558459758758545, 0.17918401956558228, -0.0849716067314148, -0.454100102186203, -0.04397144541144371, -0.3000049293041229, -1.7802939414978027, -0.7983825206756592, -2.5465900897979736, -0.009908827021718025, -1.2908704280853271, -0.5941389203071594, 0.1434621959924698, -0.257352739572525, 1.358336091041565, 0.21961842477321625, -0.06460943073034286, 2.171987771987915, -2.3252623081207275, 0.07401087135076523, -2.2763683795928955, -1.147775411605835, -0.036657124757766724, -2.273061513900757, -0.7307223081588745, -0.4726817011833191, 0.4349653720855713, -0.24327592551708221, -0.06613542884588242, -0.5492988228797913, 1.064678430557251, 2.345263719558716, -0.4237343668937683, -0.33428677916526794, -0.32627585530281067, 0.8332931399345398, 0.28617244958877563, 0.2511192858219147, -0.1238541305065155, -0.002621010411530733, 0.5774677395820618, -0.3436806797981262, 0.7624788284301758, -0.3375602066516876, -1.739920973777771, -0.20886953175067902, -0.2816256880760193, 1.232630968093872, -1.2678278684616089, 0.7018022537231445, 1.6507625579833984, -0.1685366928577423, 1.040298581123352, -0.5234507322311401, 0.8729619383811951, 0.971400797367096, 0.882211446762085, 1.0290924310684204, -0.36015522480010986, -1.0037730932235718, 0.911191463470459, 0.8230652809143066, -2.0954320430755615, -2.2884438037872314, 1.5297050476074219, 0.9958742260932922, 1.751305341720581, -0.8965188264846802, 0.28483548760414124, 2.6768898963928223, 0.12670807540416718, 2.07619047164917, -0.46876218914985657, -2.826463460922241, -0.4943554401397705, 0.29751792550086975, -0.8511837124824524, -0.2642710506916046, -1.1642699241638184, -0.6306318640708923, -0.7181330323219299, 0.5603678226470947, -0.664404571056366, -0.10924503207206726, 0.18991944193840027, -0.42091941833496094, 2.978651523590088, -1.2295323610305786, 2.858769178390503, 1.4892637729644775, 0.33450984954833984, -0.13063499331474304, -1.7591760158538818, -1.1663663387298584, -0.12473434209823608, -1.0965511798858643, 0.007678902707993984, -0.9190114140510559, -0.8081883192062378, 0.286040723323822, -0.8091283440589905, -0.6040881276130676, 0.9994192719459534, -0.09797000885009766, -1.468069314956665, 0.48661962151527405], [-0.797457218170166, -1.074204444885254, 9.108908852795139e-05, -0.2785177230834961, -1.0058929920196533, 2.129823923110962, -0.9279007911682129, -2.399190664291382, 2.403568983078003, -0.8856048583984375, 3.000831365585327, -0.5072355270385742, -1.0496290922164917, 0.9306355118751526, -0.6757861375808716, -2.2200236320495605, -0.3697567880153656, 0.18831059336662292, 0.5338422060012817, -0.6856564879417419, -0.3638867437839508, -0.09503495693206787, 0.2116144746541977, -0.0010289129568263888, -0.6588094234466553, -0.34180063009262085, -0.3029975891113281, 0.3781464099884033, 0.806204617023468, 1.789428949356079, 1.6568964719772339, -1.0992600917816162, 0.19945311546325684, -0.9881213903427124, -0.6978269815444946, -1.1190214157104492, -0.3923565149307251, 0.11543557047843933, 0.0763976126909256, 0.24813276529312134, 0.047304391860961914, -0.6301708817481995, 0.7666731476783752, -0.7052928805351257, -1.5170315504074097, 0.20908746123313904, -0.3945888876914978, -0.4919438064098358, -2.3655965328216553, 0.4931361675262451, 0.04959923028945923, -0.1748768538236618, 0.9473580718040466, 0.6519460082054138, 1.0654748678207397, -0.44096967577934265, 0.9478619694709778, 0.3731296956539154, -0.25185370445251465, 0.5165481567382812, 0.018846066668629646, -0.4803442358970642, 0.06428267061710358, -2.8886423110961914, -1.0173728466033936, 0.007147049065679312, 2.482837200164795, -1.358635663986206, -0.21871618926525116, 0.5195236206054688, 1.2495263814926147, 1.022234559059143, 0.15206797420978546, 0.42840391397476196, -1.1155023574829102, 0.3383217751979828, 1.8717596530914307, 0.36219292879104614, -1.1459929943084717, 0.5671872496604919, -0.4991879463195801, -0.0756910964846611, 0.6872265338897705, -0.4654274880886078, 1.7422304153442383, 0.6522485613822937, 0.03099631518125534, -0.2261994183063507, 1.012841820716858, 0.4823542833328247, -1.146989107131958, 0.6251070499420166, 0.49526315927505493, 0.33473318815231323, -0.6981291770935059, -0.6243926882743835, 2.151431083679199, 0.3108273446559906, 0.5030062794685364, 0.054196372628211975, 0.028326209634542465, -2.2690298557281494, -0.0661228597164154, -0.06505220383405685, -1.1497443914413452, -2.2802202701568604, -0.7138581871986389, -0.2511652708053589, 1.676200032234192, -0.7328425049781799, -0.2771410644054413, 0.9601645469665527, -0.382865846157074, 0.5334493517875671, -0.5239964723587036, -0.25689220428466797, -0.3276955485343933, 0.23774151504039764, 0.4858243465423584, -0.6612398028373718, -1.354270339012146, -1.326528549194336, 0.2549090087413788, 0.22549691796302795, 0.917526125907898, -0.703357458114624, 1.1138770580291748, 0.9911666512489319, -0.9510107636451721, -1.7203562259674072, -0.664286196231842, 0.7802907228469849, -0.40944573283195496, -0.2183408886194229, -0.909561276435852, 1.2678062915802002, -0.15068186819553375, -0.16965045034885406, -0.8639549016952515, 2.2912302017211914, -0.6096851825714111, -0.488720566034317, -0.6687690615653992, -1.13117516040802, 1.4318585395812988, 0.19853906333446503, 0.24515002965927124, 0.7487114071846008, 0.915703535079956, -0.09301898628473282, 1.365260124206543, 0.07339832186698914, 0.18306618928909302, 0.36521968245506287, 0.4785499572753906, 0.011337402276694775, 0.9621612429618835, -1.887220025062561, -1.504413366317749, 0.2509433329105377, -2.343566656112671, -0.10977382957935333, 1.134103536605835, -1.1064285039901733, -0.17871536314487457, 0.5491750836372375, -0.6682102680206299, -1.6333390474319458, 0.5386404991149902, 0.6811342835426331, 0.22787407040596008, -1.118241310119629, 0.9843866229057312, -0.1965607851743698, -1.010443925857544, -0.4832574427127838, -1.365176796913147, -0.9990127682685852, -0.24436397850513458, -0.4030347466468811, -1.508846402168274, 0.19414904713630676, -0.3399510979652405, 0.12133808434009552, 2.282456636428833, 0.8819345831871033, -0.9290146827697754, 0.07924669981002808, 1.7722651958465576, 0.6085695028305054, 0.8292487263679504, -0.5187294483184814, 1.714972734451294, 1.3896564245224, -0.46999648213386536, 1.0494608879089355, -1.3289246559143066, 0.42572933435440063, -0.28462207317352295, 1.6449708938598633, 0.1560637652873993, -0.5820710062980652, -0.6109750270843506, -1.2185876369476318, -1.041146159172058, 0.5424464344978333, -2.6921732425689697, 1.1291780471801758, -0.45352625846862793, -0.809402346611023, -0.07618749886751175, 0.5440325140953064, 0.4895420968532562, -0.9630099534988403, -0.9378494620323181, 1.6499603986740112, -1.034777283668518, 2.093684196472168, -0.7554271817207336, -0.023265864700078964, -0.7960824370384216, 0.549806535243988, 0.19178907573223114, -0.16447791457176208, -1.2609624862670898, 0.9834927916526794, -0.8119951486587524, 0.9397545456886292, -1.004126787185669, -0.5343040823936462, 0.8139660954475403, 0.9881961345672607, -0.714227020740509, -0.5385284423828125, -1.2997897863388062, -0.07873518019914627, -1.9819931983947754, -0.3690396547317505, -0.8189272880554199, 0.17977434396743774, 0.11406891793012619, 0.05954784154891968, 0.2641511857509613, 0.0033379991073161364, 0.2518934905529022, -0.9232701659202576, -0.8487716317176819, 0.15781311690807343, 1.1881946325302124, -1.5884771347045898, 0.2885097861289978, -0.01795896887779236, -0.4150800108909607, 1.0333884954452515, 0.9555339813232422, 0.46997585892677307, 0.6493104100227356, 0.8231334686279297, 0.42309099435806274, 0.7958706021308899, -0.3297519087791443, -1.464363932609558, -1.139256238937378, 0.26856860518455505, -1.8468796014785767, 0.4494723081588745, 1.760028600692749, -0.14545023441314697, -0.12874872982501984, 0.8668702840805054, -0.03112390823662281, 0.9055467247962952, 0.18151435256004333, 1.3732205629348755, 0.9513992071151733, 1.7731854915618896, -0.058849502354860306, -0.009645741432905197, 0.3425111472606659, -1.5927977561950684, -0.5352789163589478, -2.367722749710083, 0.07213771343231201, -0.9150263667106628, -1.3059183359146118, 0.2657221257686615, -0.32908061146736145, -0.4073447287082672, 0.6788931488990784, 1.03218412399292, -0.3048476278781891, 0.3054639399051666, 1.3388890027999878, -0.7387722134590149, -0.3568298816680908, -0.5292669534683228, -0.6366125345230103, 2.3914780616760254, -3.0570101737976074, 0.7864110469818115, -0.2508850395679474, 1.1151812076568604, 0.14119182527065277, -0.23845887184143066, 0.06515905261039734, 1.1396067142486572, 0.971941351890564, -0.4030084013938904, -0.4272651970386505, -0.4084028899669647, -0.6454072594642639, -0.8171524405479431, -0.7696536779403687, -0.5297616720199585, 1.3637784719467163, -0.36332571506500244, 0.5507964491844177, 0.6500033140182495, -1.1609848737716675, 1.6390116214752197, 2.918290376663208, 0.41583356261253357, 0.31780683994293213, -0.8276172280311584, -0.1345822960138321, 1.637755274772644, -2.4091882705688477, -1.139633297920227, 0.5889459848403931, 0.17372722923755646, 0.26279857754707336, 0.5181160569190979, 0.8860124349594116, -1.1209629774093628, -1.482414722442627, 0.9671916365623474, -0.9883860349655151, 0.3079230487346649, -1.003579020500183, -0.0767488032579422, -0.6305727958679199, 1.4324525594711304, -1.251448631286621, 0.46026694774627686, 1.0849275588989258, -0.37271609902381897, -1.2660070657730103, -0.9825054407119751, 0.07267264276742935, -1.9962087869644165, 1.2505385875701904, 0.8122924566268921, -0.08762579411268234, 0.6582188010215759, -1.3618196249008179, 1.2339088916778564, -0.3661805987358093, 0.8625472187995911, -0.014190921559929848, -0.5714189410209656, -0.907837450504303, 0.13322795927524567, -2.107002019882202, 1.4705886840820312, 1.1371192932128906, -0.06508497893810272, -0.24524185061454773, 1.2916971445083618, -1.2086260318756104, -1.6430100202560425, -0.9087180495262146, -2.0017433166503906, 0.7526959180831909, -1.0868737697601318, 0.36634865403175354, 0.13096550107002258, 1.8867722749710083, 0.2513749599456787, -2.040717124938965, -1.4730582237243652, -1.3426011800765991, 2.1583821773529053, 0.7722413539886475, -0.12252017110586166, -0.6969181895256042, 1.2690057754516602, 0.5150097608566284, -1.7434337139129639, 0.6843283772468567, -1.1552026271820068, 1.2538412809371948, -1.278761386871338, -0.9309453964233398, -2.0471389293670654, -1.4302351474761963, -2.0404255390167236, -0.03912689909338951, 0.2680457532405853, 1.6799017190933228, 1.642942190170288, 0.49053144454956055, -0.3123757541179657, 0.5919497609138489, -0.4697517454624176, -0.41497641801834106, -0.12203352153301239, 0.9162152409553528, 0.1523444801568985, -0.23328500986099243, 0.6907007098197937, -0.7848935723304749, 0.4457015097141266, -1.8074082136154175, -0.4163886606693268, -0.743384838104248, 0.384405255317688, 0.8650473952293396, 1.0846011638641357, -1.0422228574752808, -0.9226441383361816, -0.35948464274406433, -1.4944407939910889, -0.06466779112815857, 0.6833170652389526, 0.2160462588071823, 0.12387055903673172, 1.2399356365203857, -1.0014461278915405, 1.2312679290771484, -1.1672091484069824, 0.3844951093196869, -0.0017187197227030993, -0.19545067846775055, -1.2598448991775513, -1.0246853828430176, -0.6936825513839722, 0.6247976422309875, 0.8496074080467224, 0.8912957310676575, -0.9071717262268066, 0.02959229238331318, -2.0056328773498535, 1.6656699180603027, 1.481224775314331, -0.8301948308944702, 0.6364877223968506, -1.6862425804138184, 0.07045871764421463, 0.037735771387815475, 0.6711994409561157, -2.0916571617126465, -1.334494948387146, 0.3156627118587494, -0.044861458241939545, -2.278839588165283, -0.4019502103328705, -0.8763607740402222, 1.095005989074707, 1.3529835939407349, 0.1771741509437561, -0.17443980276584625, -0.7638692855834961, -0.7093124985694885, 0.13805048167705536, -1.569383144378662, 1.4444671869277954, 0.6374260783195496, 1.3238753080368042, 0.7805582880973816, 0.283412903547287, 0.5246787667274475, -1.0590713024139404, 1.3763272762298584, 0.3608452379703522, 0.2632124423980713, -0.4937724173069, -0.558108389377594, -0.2366846352815628, -0.36567431688308716, -0.5396226644515991, 0.3055989444255829, -0.9986773133277893, 0.6545743942260742, -0.0416974201798439, -0.07252571731805801, 0.4655676484107971, 2.0297460556030273, -1.1421977281570435, 0.7752934098243713, -0.828921914100647, -1.25343918800354, -0.10236073285341263, -1.5935163497924805, 0.955322802066803, 0.05822158232331276, -0.9244964122772217, -0.605974555015564, 0.386259526014328, -0.8567976951599121, -1.217715859413147, 0.6528700590133667, 0.5709842443466187, -1.2908564805984497, 0.31248632073402405, -0.7109171152114868, -1.4536417722702026, -0.444019079208374, 0.3434397578239441, 1.2695503234863281, 1.6369893550872803, -0.03390318155288696, 0.17412126064300537, -2.269050121307373, -0.003931072074919939, -0.7383111119270325, 0.3905172049999237, 1.3483233451843262, -0.6229525208473206, -0.23269687592983246, 1.1092822551727295, 0.4700061082839966, 0.4525134563446045, 0.8925303220748901, -0.18948052823543549, 0.715395450592041, -0.7099483013153076, 0.468160480260849, 1.31520414352417, 0.21134226024150848, -2.022308349609375, -0.19180363416671753, -1.1363439559936523, -0.4289369583129883, 0.702976405620575, 1.8537381887435913, -1.090001106262207, 0.460061639547348, 0.36747464537620544, 1.366824984550476, -0.5618282556533813, -0.45583242177963257, 0.389649361371994, -0.7711344361305237, 1.3753676414489746, -0.6889775991439819, 0.22911447286605835, 1.37142813205719, -0.27309638261795044, 0.575179934501648, 0.2890591323375702, -0.13085083663463593, 0.035006437450647354, 1.8324379920959473, -0.39740148186683655, -1.6746565103530884, -0.20411010086536407, 2.217108964920044, 0.9112471342086792, -0.5781928896903992, -0.1690039187669754, 0.12775003910064697, -0.7548730373382568, -0.3048531115055084, 0.5144781470298767, 0.43588942289352417, -1.8372982740402222, 0.9997186660766602, -1.1061744689941406, 0.005171084310859442, -1.5185580253601074, -2.2141036987304688, 0.7991399168968201, -0.5265865325927734, -0.2942267656326294, 0.71880704164505, -0.928091287612915, 1.4850884675979614, -0.02990223467350006, 1.3596018552780151, -1.0831865072250366, 1.3580520153045654, -1.2472496032714844, -0.052305180579423904, -0.960027277469635, 1.1182001829147339, -0.19186237454414368, -1.8008875846862793, 0.7101433277130127, -0.44162148237228394, 0.4320955276489258, 0.023025186732411385, -1.040848970413208, 1.4451723098754883, -1.1098719835281372, 0.47642552852630615, 0.49575257301330566, 0.48904290795326233, 1.5767860412597656, -1.8355073928833008, 0.7181735634803772, 0.1372866928577423, 0.5592090487480164, -0.23285944759845734, 0.8489871621131897, -0.15563815832138062, -0.013869873248040676, -2.064845085144043, 0.5803598165512085, -1.3028377294540405, 1.2418676614761353, 0.892030656337738, -0.27134808897972107, 1.7783750295639038, -0.7284557223320007, 0.43342307209968567, -0.4074118733406067, -1.454930067062378, 1.3950920104980469, 1.4653195142745972, 0.8178667426109314, -0.0031158176716417074, -1.231553554534912, 0.22164897620677948, -0.7364345192909241, -1.913608431816101, 0.47600963711738586, -0.2551949620246887, -1.9381526708602905, -0.5510814189910889, -1.6620017290115356, 0.9946539402008057, 0.43498149514198303, 0.42460399866104126, 0.4075331687927246, -0.002064976142719388, 0.7708310484886169, -0.15112373232841492, -1.3505083322525024, -1.6791435480117798, -1.048686146736145, -2.103853464126587, 1.3215692043304443, 0.36036866903305054, -0.9704263806343079, -0.45254507660865784, 0.14261652529239655, 0.1556195169687271, 1.1318142414093018, 0.3870878517627716, 0.2631687521934509, -0.30604398250579834, -1.058766484260559, 0.4675016701221466, -0.3542185127735138, 0.07637718319892883, 1.1498799324035645, -0.05730894207954407, -0.14102806150913239, -0.18713529407978058, -0.5987069010734558, -0.26037710905075073, 1.3563443422317505, -1.500161051750183, 0.8090693354606628, 0.43200910091400146, -2.160432815551758, 0.45374658703804016, -0.4640561044216156, 1.451353907585144, -1.5601892471313477, 0.4848995506763458, -1.3776271343231201, 1.0720678567886353, -1.3616430759429932, -0.4615707993507385, -0.8189581036567688, -0.4010275900363922, -1.2016013860702515, -1.4584574699401855, -0.9786431789398193, 0.7243945002555847, 0.5754213929176331, -0.39946529269218445, 0.07296387851238251, -0.6941321492195129, -1.2569576501846313, 0.7539427876472473, 0.2732617259025574, -0.6926500797271729, -0.8302037119865417, -0.27617347240448, -1.255900263786316, 0.97586590051651, 0.4546928107738495, -1.1598632335662842, -1.0824319124221802, 0.4500882625579834, 0.2333231121301651, 0.5437117218971252, 0.11808820813894272, 0.7917260527610779, -0.3231185972690582, -0.07894998788833618, 0.1373950093984604, -0.04063679277896881, 0.021620482206344604, -1.4366183280944824, 0.2016068547964096, 0.656718909740448, -0.6022453904151917, 0.6821706295013428, -0.337062805891037, 0.18767701089382172, -0.35855627059936523, -3.4519994258880615, 0.5122241973876953, 0.3716377019882202, 0.34541580080986023, -0.737884521484375, 0.8321642279624939, 0.0547959990799427, -1.8653230667114258, -0.8155869245529175, -1.2259645462036133, -1.5844515562057495, 0.1481350213289261, -0.08141245692968369, -0.8263819217681885, -0.44178229570388794, -0.4291737377643585, -0.5761839747428894, 0.44226109981536865, -0.26215386390686035, 0.062041208148002625, 0.3450446128845215, -0.5837826132774353, 0.13561809062957764, 0.3291441798210144, -0.2386743426322937, 0.21225354075431824, 1.5270487070083618, 0.44341957569122314, -0.6727331876754761, 1.2101536989212036, -2.734675168991089, -0.677154004573822, 1.9989739656448364, -0.4468039572238922, 1.2933403253555298, -0.055314913392066956, -1.699701189994812, 0.48202580213546753, -1.6851121187210083, 0.23209728300571442, -0.017601503059267998, 0.9989414215087891, -0.5310031175613403, 0.6243873834609985, 1.5946621894836426, 1.40192711353302, 1.5213109254837036, -0.1462223082780838, 2.072497844696045, -0.586883544921875, -0.2925354838371277, -1.738046407699585, -0.10868432372808456, 2.166827917098999, 2.1074717044830322, -0.018369100987911224, -0.4530487060546875, 0.8835806846618652, 0.7419207096099854, 0.37568381428718567, 0.26237753033638, 1.1799100637435913, 1.1665971279144287, 0.49489277601242065, 0.8424950838088989, 0.719736635684967, 1.1821882724761963, 0.6834218502044678, 0.8452038764953613, 1.9014708995819092, 0.6518701910972595, -0.19041956961154938, 0.0029417595360428095, 0.868692934513092, 0.9306963682174683, 0.5506453514099121, 0.14737187325954437, -0.27562186121940613, -0.8092196583747864, 0.5749082565307617, -0.7598934173583984, 0.016764750704169273, 1.9168956279754639, 0.6286736130714417, 0.3637179732322693, -1.1011269092559814, -1.272600531578064, 1.0675251483917236, 0.6698278188705444, -0.2682836651802063, 0.8943357467651367, 0.9818934202194214, 3.5869054794311523, 2.4055497646331787, -0.2716832756996155, -0.12005589157342911, 1.4320259094238281, -1.089184284210205, 0.20965959131717682, 1.0263171195983887, -1.1174672842025757, 1.6749550104141235, -0.8356404304504395, -0.4028763473033905, -0.5403358936309814, -0.13774101436138153, -0.07572121918201447, -0.7441096305847168, 0.004741641692817211, 1.9340945482254028, 0.18189290165901184, 2.35402250289917, 1.029387354850769, -0.17660066485404968, -0.5500407814979553, 0.23110072314739227, -0.829938530921936, 1.3553646802902222, -2.555173873901367, 0.7338775396347046, -0.5699806213378906, 1.1606074571609497, -0.6878556609153748, -0.7740936279296875, -1.8373806476593018, -1.2331477403640747, 0.556412935256958, -0.8381395936012268, 0.234805628657341, 1.1302878856658936, 0.5872989892959595, -0.2682080566883087, -0.13204063475131989, -1.0290977954864502, -0.9893971681594849, -0.5447967052459717, -1.0644114017486572, 1.6691864728927612, -0.16940240561962128, -0.22642290592193604, -1.0249933004379272, -0.8078067898750305, -0.22336719930171967, -0.5551301836967468, -0.3554686903953552, 0.6668257117271423, -0.7055898308753967, 1.0444984436035156, -1.1901750564575195, 1.0668983459472656, 0.2275337278842926, 0.8226436972618103, 2.2888123989105225, -0.5339850783348083, 0.8346447944641113, -1.5178256034851074, 0.6190154552459717, 1.3706539869308472, 1.1594107151031494, 0.22038938105106354, 0.22024264931678772, -0.18923789262771606, 0.3540738821029663, -0.10991328209638596, -0.8475425839424133, -0.9202513694763184, -0.14967840909957886, -0.8385072350502014, -1.318727970123291, -0.0014332829741761088, 0.9026951193809509, -0.4441186487674713, 1.0304453372955322, -0.9640896916389465, 0.3347318172454834, 0.9272751808166504, 0.48280540108680725, -0.9652809500694275, -1.0786209106445312, -1.162855625152588, 0.40818193554878235, -0.7451217174530029, 0.45764395594596863, 1.1263742446899414, 3.0015053749084473, -1.1834965944290161, -1.9414924383163452, 1.5344065427780151, -1.6312843561172485, -0.812993049621582, -1.2055799961090088, -1.9160475730895996, 0.15117183327674866, -0.4264441728591919, 0.9832702875137329, 1.2256392240524292, 0.9287625551223755, 1.0002375841140747, 0.6700559854507446, 1.3055837154388428, 2.315692901611328, 0.8610051870346069, 1.8265990018844604, -0.2938564121723175, -0.881874680519104, -0.1775611788034439, 0.2623896598815918, -3.849877119064331, 0.09067225456237793, 0.43721479177474976, 0.9814974069595337, -0.18551094830036163, 0.22882196307182312, -0.0827912911772728, -0.8674901723861694, 0.21722598373889923, -2.7118656635284424, -0.10083901882171631, 1.3054251670837402, 0.5631201863288879, -0.1873725801706314, -0.4965112805366516, -1.032239317893982, 0.837946891784668, -0.20522896945476532, -0.3326074481010437, 0.1946326196193695, 0.3430604934692383, -0.14906226098537445, 0.5953000783920288, -1.8645384311676025, 1.6522536277770996, 0.01922077126801014, 0.9719969630241394, -0.5205548405647278, -0.9592192769050598, -0.136392742395401, 0.8964617252349854, 0.5138481259346008, -0.2691318988800049, 0.6258382201194763, 0.6185131669044495, -0.011596770957112312, -0.2218737155199051, 2.0423529148101807, -1.5139853954315186, 1.2267639636993408, -1.7758375406265259, 1.2950936555862427, -0.6439417004585266, 0.02157791145145893, -1.0481672286987305, -0.996836245059967, 0.5118767619132996, 1.9040610790252686, 0.4289379119873047, -0.4259067177772522, -0.26420891284942627, 0.292982816696167, 1.3424420356750488, -1.6923376321792603, -1.512573003768921, 1.0778366327285767, 0.4806455969810486, -1.0564521551132202, 2.3658456802368164, -1.0329022407531738, 0.3753701150417328, 0.2621889114379883, 0.4947124719619751, 0.18389759957790375, 0.6008721590042114, 0.336330771446228, 1.4020110368728638, -1.778054118156433, -1.1046489477157593, 0.5787612795829773, 0.026077838614583015, 0.3214821517467499, 0.5004726052284241, 0.2598010301589966, -0.9806968569755554, 0.3813016414642334, 0.18774273991584778, -2.1846988201141357, -0.05289626866579056, -0.7526184916496277, 0.6754882335662842, 0.7866370677947998, 0.3427392244338989, -1.8708012104034424, 0.8837572932243347, -0.10415646433830261, 0.7304154634475708, -2.320683240890503, 1.224656343460083, 0.41704922914505005, -0.21191921830177307, 0.012422981671988964, 1.066324234008789, -1.0794323682785034, -2.4766814708709717, -0.5243384838104248, -0.1324853152036667, 1.0413450002670288, -0.7496910095214844, -0.40937188267707825, 0.828004002571106, 0.4961049258708954, -0.03894617035984993], [0.28765547275543213, -0.547247052192688, -0.2796365022659302, 1.2715277671813965, -0.6848104596138, -1.2377424240112305, -0.7470383644104004, 0.21515768766403198, 0.5865095257759094, 0.45558273792266846, -0.3904525339603424, 0.46750760078430176, -1.210642695426941, 0.13764724135398865, 0.7151881456375122, 1.1996750831604004, -0.9987401962280273, -0.4934079349040985, 0.5073111653327942, -0.8449316024780273, 0.10167432576417923, -1.4113324880599976, -0.5607782006263733, 0.03915254771709442, 0.9623205661773682, 2.1962673664093018, 0.5086421966552734, 0.5289273262023926, -0.019698016345500946, -0.8447039127349854, 0.6411106586456299, -0.5831518173217773, 1.112474799156189, 0.895622730255127, 0.02238881215453148, 0.11258834600448608, -2.189091205596924, 0.2517416179180145, 0.19850483536720276, 0.5929768085479736, 1.4653081893920898, 0.1704566329717636, 0.6364151835441589, -1.6401376724243164, 0.3623510003089905, -0.5634802579879761, -0.3737911581993103, 1.2186869382858276, 3.3106179237365723, -0.6107373237609863, -0.058531492948532104, 1.2434613704681396, -0.14690035581588745, -1.1272128820419312, -0.6463898420333862, 0.546094536781311, 0.2677677273750305, -0.5186023116111755, 0.15948420763015747, -1.071694254875183, -1.3226706981658936, 2.108734130859375, 0.7291480898857117, -0.09031809121370316, 0.9436036944389343, -0.2849499583244324, 0.1559629738330841, 2.539781332015991, -1.0154893398284912, 0.7623398900032043, -0.9412465691566467, 0.5975781679153442, -0.6513245701789856, -0.611041247844696, 1.3888963460922241, 0.15727567672729492, -0.4709182381629944, -0.25798124074935913, -1.0820411443710327, 0.49962806701660156, 1.223954439163208, -0.9509431719779968, 0.8826274275779724, -0.3003074526786804, 0.8263323903083801, 0.6531715989112854, -0.6004475951194763, 1.3065106868743896, -1.0002261400222778, 1.033913493156433, -0.0803207978606224, 1.5995303392410278, 0.3729863464832306, 1.8831068277359009, -0.4933696985244751, -0.7405746579170227, 1.6378605365753174, -2.4381675720214844, -1.9689428806304932, 0.5069293975830078, -0.28961634635925293, -1.123470664024353, 0.0993356704711914, 0.9904634952545166, 0.5048606991767883, 0.18528902530670166, -0.05059284344315529, 1.6224650144577026, 1.235592246055603, -0.35103926062583923, 1.3212422132492065, -0.2237412929534912, -1.10160493850708, -0.6231304407119751, 0.177379310131073, 0.6170881986618042, 1.086378574371338, -0.6152490377426147, -1.128428339958191, 1.6283485889434814, 0.7196769714355469, 0.572627067565918, -0.8553898334503174, -0.08452077209949493, 1.539119005203247, 0.5297728180885315, 1.3821383714675903, 0.496576726436615, -1.1337522268295288, -0.6027059555053711, -0.7002980709075928, 1.3099186420440674, -0.12253046780824661, -2.481658697128296, -1.2241647243499756, -1.0188496112823486, -1.0222645998001099, 0.6151067018508911, 2.5212016105651855, 2.1519932746887207, -0.22239293158054352, -0.7709286212921143, -1.9016509056091309, 2.0465610027313232, -1.142529845237732, 0.39940470457077026, -1.1247378587722778, 0.04881215840578079, -0.9337283372879028, -1.529536485671997, 0.4489127993583679, 2.292311191558838, 0.08331906795501709, 0.3046976327896118, -0.17890013754367828, -0.07213182747364044, -0.9582808017730713, 0.46033474802970886, -0.1653245985507965, 1.082245945930481, 1.2031418085098267, -1.3980998992919922, 0.2705782353878021, -0.7403450608253479, -0.4928853511810303, 0.7289246320724487, 0.48239895701408386, 0.03437816724181175, -1.4309548139572144, 0.1460648626089096, -0.148442804813385, -0.3950807452201843, 0.6523862481117249, -1.4640368223190308, -0.3245055675506592, 0.3785453140735626, 1.3239496946334839, 0.10066604614257812, 1.2255691289901733, -0.6890987157821655, 0.03860452398657799, -0.49062782526016235, 1.2433809041976929, -2.060871124267578, -0.9117738604545593, -0.4819431006908417, 0.668038547039032, 1.0065927505493164, -0.2384105771780014, -1.3838441371917725, 0.3874814808368683, -1.0442571640014648, -0.7465662360191345, 1.0244516134262085, 0.7420348525047302, 0.42656436562538147, 0.4525502324104309, 0.46612560749053955, -1.6366958618164062, -0.0004277346015442163, -1.3848047256469727, -1.5852537155151367, 0.5532844066619873, -0.25990647077560425, 0.13940708339214325, -1.8150393962860107, -1.8202264308929443, 0.8705924153327942, -1.1472773551940918, 0.7495391964912415, -0.23795302212238312, 0.40422120690345764, 0.26031389832496643, 1.4878002405166626, 0.24639561772346497, -0.09611924737691879, -0.5522557497024536, -1.6023619174957275, 1.079065203666687, 0.20736004412174225, 1.2767220735549927, -1.2962076663970947, -1.3029425144195557, 1.7197325229644775, 0.7575180530548096, -0.638808012008667, 1.1331974267959595, 1.4595906734466553, -0.1451711505651474, -0.5681567192077637, 0.35089758038520813, 0.17667557299137115, 1.226298213005066, 1.9976749420166016, 2.2054338455200195, -0.5205356478691101, 0.0510048046708107, 0.661581814289093, -0.5712074041366577, -0.46040287613868713, -0.1262563019990921, -2.2978954315185547, 1.1466807126998901, -1.7572917938232422, -1.1623051166534424, -2.096104383468628, 1.0749080181121826, -0.677466094493866, -1.0076987743377686, -0.6008927822113037, -1.041155219078064, -0.3361702859401703, -1.7968159914016724, -0.24065928161144257, 1.1443485021591187, 0.7218733429908752, 1.258915901184082, -1.1175203323364258, 1.0611379146575928, -1.311974048614502, 0.9483081102371216, 1.1212387084960938, -2.0482704639434814, -0.43723779916763306, -0.556678056716919, 0.341989129781723, -1.2818541526794434, -0.8036757707595825, 1.6434918642044067, -0.4984172284603119, -0.591408908367157, -0.3341602385044098, -0.7027785778045654, -2.0796680450439453, 0.9622127413749695, -0.2827819883823395, 0.2854014039039612, -0.05322961136698723, 0.7514365911483765, -1.431127905845642, 2.0581181049346924, -0.5838460326194763, 0.1818210482597351, -0.1068071499466896, -0.7475956678390503, -0.6862960457801819, 0.35839954018592834, 0.5269672274589539, 0.23206773400306702, 0.6668403744697571, -0.1335810124874115, -0.5146476030349731, 0.6823920011520386, -1.0069196224212646, -0.49007755517959595, 0.5749771595001221, -1.7643628120422363, 0.13090600073337555, -1.6115916967391968, 1.0090222358703613, 0.03529377654194832, 0.06333693116903305, 1.2147471904754639, -0.1778648942708969, -0.8412132859230042, 0.611768901348114, 0.9778822064399719, -0.4204411506652832, -1.1304234266281128, 0.3717314898967743, -0.9905208945274353, -1.7041059732437134, 1.0206754207611084, 1.0613571405410767, -0.06073456257581711, -0.8665977120399475, 1.571366548538208, -0.6885718107223511, -0.9665393233299255, -1.00316321849823, 1.333113431930542, -0.08943785727024078, 1.033855676651001, -0.828761875629425, 0.5810192823410034, 1.0190285444259644, 0.06473038345575333, 1.1707992553710938, 0.5332514643669128, 1.6092166900634766, -0.18745674192905426, 1.7113680839538574, 1.4601017236709595, -2.251333236694336, 0.7200701832771301, 0.091941699385643, -0.1915821135044098, 0.4642176628112793, -0.18230685591697693, 0.4145430326461792, -0.06044508144259453, -0.6660519242286682, -0.34674084186553955, -0.032103005796670914, -0.5859976410865784, -0.9540073871612549, 0.9093040823936462, -1.0778530836105347, 1.6469448804855347, -1.9630497694015503, 0.41049930453300476, -0.605342447757721, -0.4853929281234741, 0.05182930827140808, 0.8164538145065308, 0.07559105008840561, -0.5136587619781494, 0.9442875981330872, 0.5180370211601257, 0.99200439453125, -0.6345319151878357, 0.8850095868110657, 0.1727374941110611, -1.7432385683059692, 0.09786080569028854, 2.4086883068084717, -0.4412590563297272, 2.012653112411499, 0.8095024824142456, 1.1122784614562988, 0.1460239142179489, 0.5957633852958679, 0.3805883526802063, -0.5515246987342834, -1.0122578144073486, 1.4179712533950806, 0.19739502668380737, -0.7464587092399597, 0.8288911581039429, -1.0048810243606567, 1.066672921180725, -0.35407134890556335, -0.10203741490840912, 0.04943817853927612, -1.1548064947128296, -1.1551353931427002, 0.3652941584587097, -1.3293323516845703, -1.943581223487854, 0.4689863622188568, -0.2512536644935608, 0.8903758525848389, -0.6931061744689941, -0.47879675030708313, 0.938459038734436, -0.6625426411628723, -0.8101658225059509, 0.9423194527626038, 0.2411920726299286, 0.5048799514770508, -1.058601975440979, 0.04115353524684906, 0.25646236538887024, 0.34655332565307617, 1.8503774404525757, -1.4099769592285156, 1.1485756635665894, -1.4530417919158936, -0.5838379859924316, -0.4860990047454834, 0.36975157260894775, 0.587620198726654, -1.1583021879196167, -1.1992336511611938, 0.14746980369091034, 0.19267898797988892, 0.7158692479133606, 0.014452754519879818, 0.2068774402141571, -0.24731598794460297, 1.506103515625, -0.40135905146598816, 0.13790708780288696, 2.576293706893921, 0.16130976378917694, 1.937928318977356, -0.014919411391019821, 0.9368400573730469, 2.6001837253570557, -2.034085273742676, 0.4334426820278168, -0.7530530095100403, 0.8183920383453369, 0.8756090998649597, 1.8049030303955078, 1.1883220672607422, 0.8198611736297607, 0.32303330302238464, 0.7872154116630554, 0.7142632007598877, -0.7936462759971619, -0.03442908450961113, -0.7658856511116028, 0.6282007694244385, 0.896952748298645, 0.14414486289024353, -0.9117660522460938, -1.9549667835235596, 1.2114770412445068, 0.40425804257392883, 0.7728996276855469, 0.6621250510215759, 0.030949430540204048, 0.7138132452964783, 0.6059560775756836, 0.6064653396606445, -0.14218300580978394, 0.06243016943335533, 0.3685448169708252, -0.012572596780955791, -1.02271568775177, -1.184720516204834, -0.536406397819519, -0.5465442538261414, 0.9682499170303345, -0.28207507729530334, 0.08084699511528015, 0.23927612602710724, -0.1469683051109314, 0.41063958406448364, -1.2486685514450073, 0.6061006188392639, -0.9171244502067566, 0.10380492359399796, -2.225210666656494, 0.3219117224216461, 0.7425600290298462, 1.1466124057769775, 0.8220835328102112, -0.12922827899456024, -0.7256941199302673, -0.5083184242248535, -0.9803803563117981, 0.2945404350757599, 1.4751945734024048, 0.3593592345714569, 1.1249988079071045, -1.3899130821228027, 1.1498035192489624, -0.003232272109016776, -0.7739366292953491, 1.1749110221862793, 0.05983347073197365, 0.6165391206741333, -0.9522579312324524, -1.6279515027999878, 0.20168498158454895, -0.6789778470993042, -1.3301905393600464, -0.5768716931343079, -0.04475310072302818, -1.0455491542816162, -0.007450784556567669, 0.027919163927435875, 0.6759628653526306, -1.454831600189209, 0.5429198741912842, -0.13297224044799805, -0.39060351252555847, 1.4242624044418335, -1.1753147840499878, 0.5458898544311523, -0.5997487306594849, -0.21840843558311462, -0.03487135469913483, -1.9469976425170898, -0.3756386637687683, 0.9886886477470398, -0.1605527102947235, 0.604953944683075, -0.6987611651420593, 1.1500144004821777, -0.7660582661628723, -1.6506510972976685, -0.63164222240448, 0.9159769415855408, -0.02627130039036274, -0.19709718227386475, -1.4207777976989746, 0.1721268594264984, 0.3191263973712921, 2.4245221614837646, 0.38457462191581726, -0.4316101372241974, -2.5275518894195557, 0.7959707975387573, 0.8231602311134338, -2.344670295715332, -0.02891514077782631, -3.019120454788208, 0.9968545436859131, -1.9368996620178223, 0.7816491723060608, -1.685266137123108, 2.049193859100342, 1.342279314994812, 0.5616140961647034, -0.18770159780979156, -0.9669628143310547, -0.06546731293201447, -0.35939836502075195, -1.2027465105056763, -0.14943312108516693, -0.04750629886984825, 0.8479631543159485, 2.09698748588562, 0.574593186378479, -1.9873727560043335, -0.8933138847351074, -0.33524975180625916, -1.2380728721618652, -0.9400386810302734, -1.7459007501602173, -0.3634205162525177, 1.2848268747329712, -0.37383630871772766, -0.590092122554779, -1.3267829418182373, -0.8295210003852844, -0.21778565645217896, -1.4505914449691772, -1.1855263710021973, 0.5109730958938599, 0.6837611794471741, 1.003800630569458, -0.03214709460735321, -0.33344149589538574, 0.2819308340549469, 0.09970872849225998, 0.7858135104179382, 1.244031548500061, 0.17833350598812103, 1.7374097108840942, -1.9890954494476318, -0.14053481817245483, -0.9304071664810181, -1.4107304811477661, -0.8370829820632935, -0.29081955552101135, 1.1913866996765137, 1.17097008228302, 0.6724545955657959, -0.3226650357246399, -2.1066079139709473, -0.2820659875869751, 1.7723604440689087, 2.164724349975586, -0.7243673801422119, -0.5290036797523499, -1.8562724590301514, 0.772746741771698, 1.586851716041565, -2.4037351608276367, 0.5281751155853271, -0.6169952750205994, -0.13770174980163574, -0.5403756499290466, -0.11626797169446945, 1.5931611061096191, 0.2855217158794403, 0.5107301473617554, -0.3899292051792145, -0.67569500207901, -1.6641960144042969, 0.4132000803947449, -1.0510691404342651, -2.0431859493255615, 1.2060450315475464, 1.15269935131073, -1.1241966485977173, -0.09066850692033768, 1.061192512512207, -0.7932201623916626, 0.36836713552474976, -1.543559193611145, -2.073593854904175, -0.27393803000450134, 0.9342794418334961, 1.72697913646698, 0.7121305465698242, 0.01770808920264244, -0.5759016871452332, 2.032201051712036, -0.544167697429657, 0.8920027613639832, 0.2592594623565674, 1.27982759475708, -0.14788296818733215, -0.5482637882232666, 0.5654090046882629, -0.05464976653456688, 0.26223039627075195, -1.216628074645996, 0.4824689030647278, -1.2064934968948364, 0.29589998722076416, 0.6030195951461792, 0.11631148308515549, -0.8013882040977478, -0.9449129700660706, 0.014363963156938553, -2.2415826320648193, 0.3019642233848572, -0.08370447158813477, 0.23880696296691895, -0.04717501252889633, -0.8252782225608826, -0.0012420944403856993, -1.2199749946594238, -1.791685938835144, -0.8909692764282227, 1.1142436265945435, -0.6324117183685303, 0.519945502281189, -0.6267938017845154, 0.24174325168132782, 2.0053329467773438, -1.3084776401519775, 0.10082655400037766, -0.6721318960189819, -1.035003423690796, 1.264549732208252, -0.9997161030769348, 0.718230128288269, -0.843829333782196, 0.7318029403686523, 0.32665276527404785, 0.09254098683595657, 1.9057347774505615, 0.7839688658714294, -1.3974107503890991, 1.2334704399108887, 1.27406644821167, -0.21383506059646606, -0.7524964213371277, -0.28382691740989685, -0.33996954560279846, 1.7640701532363892, -1.0769128799438477, -1.8543087244033813, 1.470719575881958, -0.2712514400482178, -0.7373687624931335, -2.0328288078308105, 0.5648409724235535, 1.1105868816375732, -0.38696417212486267, 0.6589888334274292, 0.6915250420570374, -1.8095951080322266, 0.3333873748779297, 0.7867462635040283, 0.7481709718704224, 0.2828156650066376, -0.06766650825738907, 0.7331975698471069, 0.06990783661603928, -0.7991870045661926, -0.6154645085334778, 2.051931858062744, -1.2794126272201538, -0.33353468775749207, -1.3679299354553223, 0.4209921061992645, -0.501917839050293, 2.3713741302490234, -0.8971161842346191, 0.16729693114757538, 0.5895654559135437, -0.5472108125686646, 0.5896012783050537, 0.23508867621421814, 0.32607316970825195, -1.5984501838684082, 0.6443496942520142, 1.5844603776931763, -0.35596054792404175, 1.0171805620193481, -1.0054517984390259, -0.3312339186668396, -0.3350115716457367, -1.4124946594238281, 0.6648131608963013, 2.3253748416900635, 1.3256489038467407, -1.2174721956253052, 0.33096596598625183, 1.953616738319397, 1.1428914070129395, -1.6484792232513428, 2.084041118621826, 1.145917534828186, -0.5426367521286011, 1.9931803941726685, -0.08926450461149216, 1.0212197303771973, 0.6873816251754761, -0.47055748105049133, -0.2199791520833969, 0.19224421679973602, 0.7605546116828918, -2.0509634017944336, -1.0032551288604736, 0.5383329391479492, -0.46306702494621277, -0.6657736897468567, -1.1218698024749756, 0.8411927223205566, -1.1584306955337524, 0.5230817794799805, 1.4413551092147827, 0.751412034034729, 0.5151596665382385, -1.1222847700119019, 1.0703630447387695, 0.9915772080421448, 1.2924206256866455, -0.05405358970165253, 0.20490850508213043, -0.27731990814208984, 1.132693886756897, 1.0975042581558228, 1.4335850477218628, -0.31791922450065613, 2.353817939758301, -1.5354235172271729, -0.9885422587394714, -1.013948678970337, -0.20949220657348633, -0.24589550495147705, -0.5225545167922974, 0.6096441745758057, 1.1113041639328003, -0.5566132664680481, -0.9866724014282227, 1.2424187660217285, 0.07388091832399368, 0.1615031212568283, 1.673986554145813, -1.2977232933044434, 0.5630442500114441, 1.3923636674880981, -0.45413658022880554, 0.5120112895965576, -0.20831076800823212, -0.9635647535324097, 0.257992148399353, 1.2072638273239136, -0.6418110132217407, 0.23629975318908691, 1.2756749391555786, -0.9617173671722412, -0.8228259086608887, 0.6634743213653564, -0.1835474669933319, -2.1440043449401855, -0.8512985706329346, 0.429475873708725, 0.4736069142818451, -0.8900632262229919, -1.7087204456329346, -1.3469188213348389, 0.2869601249694824, 0.3812807500362396, -0.38082021474838257, 1.0377697944641113, 1.1379520893096924, -1.2108656167984009, 0.6947896480560303, -0.1115357056260109, -2.029228925704956, 0.13522885739803314, 1.1620874404907227, 1.3533289432525635, 0.3234063386917114, -0.3375309407711029, 0.5099643468856812, 0.3195813000202179, -0.09606817364692688, -2.18634295463562, -0.32824599742889404, -0.5817828178405762, -0.4794403314590454, 0.593434751033783, 0.8091644644737244, -0.7281326055526733, 0.5404573082923889, 1.4555878639221191, 0.23299644887447357, 0.3660903573036194, -1.8555221557617188, -1.4323434829711914, -0.6960707306861877, -0.8764477968215942, -1.493201494216919, -0.7807921767234802, 1.2502977848052979, -0.7901996970176697, 1.6051862239837646, 0.4025719463825226, 1.097325086593628, 1.5133624076843262, 0.3837563991546631, 1.0884429216384888, 0.18804757297039032, 0.6478138566017151, 0.40801864862442017, -0.5952418446540833, 0.01922440156340599, -0.4529520571231842, -0.7378247380256653, -1.1485601663589478, 0.060657765716314316, -0.21810656785964966, 0.4545179307460785, -1.3444870710372925, -1.2944995164871216, -0.1837070733308792, 2.13641095161438, -1.4642362594604492, 0.7796030044555664, 1.6003093719482422, 0.2651236057281494, -0.4742245674133301, -0.34510931372642517, -0.03607172146439552, 0.7431816458702087, -0.2068587690591812, 0.589850127696991, -1.1933262348175049, -0.39482709765434265, 1.8699126243591309, 0.21823835372924805, 0.9677493572235107, 1.301277756690979, -0.6005237698554993, 1.2037769556045532, -1.7064417600631714, -1.2264412641525269, 0.5314772129058838, 1.3927133083343506, -2.3832268714904785, -0.25908803939819336, 1.554105520248413, -0.5743845701217651, 1.1447416543960571, 1.9163475036621094, -0.1946362406015396, -0.6663780212402344, -0.30987533926963806, 0.6004626750946045, 0.20608067512512207, -1.7761796712875366, -0.6311711072921753, 1.7723151445388794, -1.4278184175491333, 0.6597063541412354, -0.9113271832466125, -0.6347437500953674, 2.305198907852173, -1.1710071563720703, -0.4214729964733124, 0.7754929065704346, -0.23432675004005432, 0.11349906772375107, -1.4002124071121216, -3.1933774948120117, 0.03476841375231743, -0.23055899143218994, 1.4843311309814453, -0.5087792873382568, -1.4871675968170166, -1.116389274597168, -0.27928122878074646, 0.5869777798652649, 1.2754508256912231, -0.0574905127286911, 0.5439314842224121, 0.6440346837043762, -1.0072879791259766, -1.5807607173919678, 0.878557562828064, -0.6594732999801636, 0.2686276435852051, 0.5407291054725647, 0.13887424767017365, 0.03253212943673134, 0.5866701006889343, 2.136199951171875, -0.4625922739505768, 0.35622456669807434, -0.10777896642684937, 1.46578848361969, -0.11527389287948608, -0.8426321148872375, -1.4955618381500244, -0.31743624806404114, 0.10957235842943192, -0.09912660717964172, 0.5996846556663513, 1.1954708099365234, -1.3817330598831177, -0.897095263004303, -0.4226285517215729, -1.0321934223175049, -0.2925511300563812, -0.7630630135536194, -0.28899261355400085, 0.715800940990448, -1.428911566734314, -0.8355502486228943, -0.7583673000335693, -0.6711382865905762, -0.27638551592826843, 0.1796966940164566, -1.6775541305541992, -0.0830526053905487, -0.3104795217514038, -1.0800914764404297, 0.4678541123867035, 0.904795229434967, -0.016683120280504227, -0.08558018505573273, -0.9937031865119934, -1.5769224166870117, 0.967931866645813, -1.2721649408340454, -0.5861803293228149, -0.691077470779419, 0.7776398658752441, 0.21677526831626892, -0.626192569732666, 1.3392722606658936, -0.2300165444612503, 0.051982227712869644, -0.36483532190322876, 0.21697898209095, -0.007715124636888504, -0.6181507110595703, -1.115525722503662, -0.5201974511146545, -0.7510015368461609, -1.091773271560669, -0.4831058084964752, 0.9489032030105591, -0.8580570220947266, 1.649527668952942, -0.9886449575424194, -0.40275076031684875, 1.7350302934646606, -0.8776208162307739, -0.9173641204833984, 1.0266852378845215, -2.809863328933716, 1.3812764883041382, 2.1791651248931885, 0.29003140330314636, 0.1517048329114914, -0.025324644520878792, 0.034815628081560135, 1.252289891242981, -0.3020966649055481, 0.3881736099720001, 0.5904451012611389, -0.7995133996009827, -1.8801888227462769, 0.6543112993240356, 0.16415411233901978, 0.3774934411048889, -1.6913702487945557, 1.1069387197494507, -0.6813842058181763, 1.0412712097167969, -0.7635905742645264], [-0.20242084562778473, 0.03030262514948845, 1.8570581674575806, 0.9904822707176208, -0.6047630906105042, -0.9873084425926208, -0.4567289352416992, 0.9635615944862366, 1.4470001459121704, 0.9962040781974792, 1.4281166791915894, 0.010868365876376629, 0.8478206396102905, -1.0484130382537842, -1.029853105545044, -1.4542361497879028, 0.6985617876052856, -1.0656397342681885, -1.3111810684204102, 0.47066569328308105, 0.6218869686126709, 1.0976402759552002, 0.6458914875984192, 1.234311819076538, -2.978558301925659, 1.0577774047851562, 2.2260265350341797, 0.6539655327796936, -0.028849473223090172, -1.179006576538086, 0.45165887475013733, -1.2034034729003906, 0.46115875244140625, -0.820193350315094, 0.9421088695526123, 0.13238751888275146, 1.146124243736267, 1.9307982921600342, -0.8110703825950623, 0.9097657203674316, 0.027603091672062874, 0.956974446773529, 1.638974905014038, 0.7029699087142944, 0.036311063915491104, -0.7868927717208862, 0.3523593544960022, 0.48849570751190186, -1.0643589496612549, 0.6275611519813538, 1.0231016874313354, -1.0178346633911133, 0.24931953847408295, -0.4196089208126068, 0.5702771544456482, 0.0516577810049057, -0.2959497570991516, -0.838680624961853, 0.7610867619514465, 0.17888052761554718, 0.3286590576171875, 0.07168328762054443, 0.14762000739574432, 0.9593058824539185, 1.3571176528930664, -0.14086465537548065, 1.0863062143325806, -0.6662416458129883, -0.15295296907424927, 1.0125136375427246, -0.2004866898059845, 0.03458835184574127, 0.1551966816186905, -0.5913617610931396, -2.3363194465637207, 1.5738239288330078, 0.1921447366476059, -1.9247077703475952, 0.14885561168193817, -1.5434138774871826, 0.3858894407749176, 0.08490631729364395, 1.2026762962341309, -1.8905980587005615, 0.025099383667111397, 0.27590349316596985, 0.5014127492904663, 1.3971097469329834, 0.6336973905563354, -0.5293293595314026, -0.36426058411598206, -0.38488468527793884, 0.730646550655365, -0.11548963934183121, -1.5286691188812256, 1.5327988862991333, -0.2936854064464569, -1.0075936317443848, 0.9537296891212463, 0.32951614260673523, -0.8988317251205444, 0.8297693729400635, -1.2702757120132446, 0.011718552559614182, 0.5007339119911194, 0.7289991974830627, -0.3538963794708252, -1.087684988975525, -0.13482584059238434, -1.0063023567199707, -0.6148479580879211, 0.043677397072315216, 0.25464048981666565, -2.1509642601013184, -2.2869532108306885, 0.30664682388305664, -0.06832676380872726, -1.2462103366851807, -0.9241352081298828, 1.3378304243087769, 0.9027881026268005, -0.4226438105106354, -0.33869919180870056, -1.4869049787521362, -0.13601988554000854, 0.2126958817243576, -0.8993262052536011, 1.916680932044983, 0.5997490882873535, -0.9940750002861023, 0.561946451663971, -2.1879994869232178, 0.1191580668091774, 1.3706504106521606, -0.2395029515028, 1.090238094329834, -1.3284939527511597, -0.21963050961494446, -1.1381263732910156, -0.8711282014846802, -0.6096252799034119, 0.2101856768131256, 0.03922519087791443, 0.5961016416549683, -0.5981395840644836, -0.7518318891525269, -2.513676404953003, 0.7737973928451538, 0.4342147409915924, -0.6832181215286255, 1.464752435684204, -0.15258555114269257, 0.4310215413570404, -1.567473292350769, -1.1226012706756592, -0.891413688659668, 0.22395846247673035, -0.007877109572291374, -0.289326012134552, -0.4837888777256012, 0.09970588237047195, -0.8667739033699036, 0.16829441487789154, -2.107550621032715, -1.8901020288467407, 0.08842294663190842, -0.025820141658186913, 0.5470287203788757, -0.2996111512184143, 1.1020725965499878, -0.6829447746276855, 0.3368810713291168, -0.16459743678569794, -1.249353289604187, -0.6185284852981567, -1.636638879776001, -0.20861048996448517, 0.8164483904838562, -0.10255255550146103, 0.2351560741662979, 0.6781044602394104, -0.1782582551240921, 0.7191174626350403, -0.6246666312217712, 0.7445127964019775, 1.1286383867263794, -0.22693680226802826, -1.1234244108200073, 0.4723707139492035, 0.0863717794418335, 0.17418643832206726, 0.9553430080413818, 1.6403493881225586, 1.8041268587112427, -0.19777709245681763, 0.9268561601638794, -0.13936102390289307, -1.5884534120559692, 1.2022322416305542, 1.6078988313674927, 0.5813643336296082, -0.8373653292655945, -1.6255940198898315, -0.7006431221961975, 1.9982945919036865, -0.9578914046287537, 0.836969792842865, 0.8065608143806458, -0.4891711473464966, 0.6116510033607483, -0.16857188940048218, -0.3779166042804718, -0.08511075377464294, 1.0131750106811523, 0.09278985857963562, 0.07344424724578857, 2.232903480529785, -0.15333831310272217, 0.4268043339252472, 0.18648716807365417, 1.0559474229812622, 1.0019830465316772, -0.2788250148296356, -1.4403799772262573, 0.25055551528930664, -2.803405523300171, -0.7747379541397095, 1.2849117517471313, 0.5668277740478516, 1.3875126838684082, -1.28377366065979, -0.8468844890594482, -0.0029905112460255623, -1.2204269170761108, 0.3395113945007324, 0.38996997475624084, 0.08966316282749176, -0.17973802983760834, -0.5655801296234131, -0.20757006108760834, 1.146754264831543, 1.0436739921569824, -0.6094216704368591, -0.9359888434410095, 1.727299690246582, -0.24102522432804108, -0.7919002175331116, -1.4844262599945068, 1.4517579078674316, 1.0961511135101318, 1.2456799745559692, -0.3124544620513916, -0.8037203550338745, 0.6083560585975647, -0.1823815107345581, -0.4944508671760559, -0.4211382567882538, -0.13611149787902832, -0.20331746339797974, 1.8247133493423462, -0.40255627036094666, -0.6296285390853882, 0.3763818144798279, 0.5117926597595215, 0.43918758630752563, -0.6850777864456177, -1.473832368850708, -0.7246513366699219, 0.5742776393890381, 0.7795464992523193, 0.4012208580970764, -1.2489360570907593, -1.1503207683563232, -1.2545901536941528, 0.028948741033673286, 0.4732169508934021, 0.8468456268310547, 1.2493510246276855, 0.8792816400527954, -0.3936994969844818, 0.5484641194343567, 0.47742849588394165, -0.6126784086227417, 0.10427861660718918, 0.9003556966781616, -0.8577674031257629, -0.2497689425945282, 1.046306848526001, 1.433591365814209, -0.7547268271446228, 0.20787875354290009, -0.9644666314125061, 0.8627861738204956, -0.20226402580738068, -1.0196208953857422, -0.6237516403198242, -1.5296565294265747, 0.26571983098983765, -0.5993924736976624, -0.5030817985534668, -0.910656213760376, 0.8502267003059387, -0.00903462152928114, 0.2512151896953583, 0.42674490809440613, 0.5655875205993652, 0.3437637388706207, -0.3121393024921417, -0.8720499873161316, -0.3022325336933136, -0.9842542409896851, -1.265054702758789, 0.4841902256011963, -0.34548407793045044, 0.06289654225111008, 0.41569826006889343, 0.7290066480636597, 1.4886869192123413, 1.0047881603240967, 0.7596660256385803, 0.23335354030132294, 0.2378425896167755, -0.26971814036369324, 1.7474188804626465, 1.0010453462600708, 0.1811925321817398, -0.8752431273460388, 0.5477434396743774, 1.214937448501587, 0.22807516157627106, 0.6589855551719666, -2.0343565940856934, 0.16942915320396423, -0.29207107424736023, -0.5190085172653198, 1.3266761302947998, 0.3165876865386963, -0.01783090829849243, -0.31316688656806946, -0.21412688493728638, 0.9694824814796448, -0.8501814007759094, -0.2857127785682678, 0.38338348269462585, 0.25225552916526794, -1.8931225538253784, -0.30592599511146545, 0.46679770946502686, -1.3541241884231567, 0.5238433480262756, -0.4786476194858551, 0.2749119997024536, 0.9131413102149963, 0.5527220368385315, -0.27784252166748047, 0.6825838685035706, -1.032497525215149, 0.7455785870552063, -0.33698341250419617, -0.29592379927635193, 1.7694475650787354, -0.39864104986190796, -0.1265484243631363, -0.2103809267282486, -0.8343149423599243, 0.7347946763038635, -0.13178548216819763, -1.2049405574798584, 0.5018392205238342, -0.10999421030282974, 0.2888733446598053, -1.8699698448181152, -0.5193127393722534, 2.4001314640045166, 1.6370995044708252, 0.9438172578811646, -2.9478535652160645, -0.1635073721408844, 1.0320916175842285, 0.8813825845718384, 1.1045912504196167, -1.1992899179458618, -0.5603852272033691, 1.3823975324630737, 0.9796116948127747, -2.53348970413208, -0.12849989533424377, 1.7094001770019531, -0.8718308806419373, 2.1878011226654053, -0.4030323624610901, -1.5656861066818237, -1.5882025957107544, 0.06818000972270966, 0.2086542844772339, 0.07587362825870514, -0.09655969589948654, -0.784559428691864, -0.4280523359775543, 0.7261922359466553, -0.5006322264671326, 1.318068265914917, 1.0793887376785278, 1.15861177444458, 2.0483624935150146, 0.3478148579597473, 0.33160635828971863, 1.5799106359481812, -0.4026375710964203, 0.19008411467075348, 0.2347130924463272, -0.4210929870605469, -0.7396348118782043, -1.0881553888320923, 0.021834848448634148, -0.7130389213562012, 0.7525089383125305, -0.2807033061981201, -0.3221477270126343, -0.17318737506866455, -1.4601517915725708, -0.27446281909942627, -1.8924784660339355, 0.06424345821142197, -0.8586984276771545, 0.8813936114311218, 0.6331171989440918, -1.1893882751464844, -0.24726547300815582, -1.7115912437438965, -0.3608587384223938, -1.4437856674194336, -2.353937864303589, -0.22352702915668488, -0.6464941501617432, -0.5831484794616699, -1.5170104503631592, -1.3188930749893188, 0.20448672771453857, 0.19200070202350616, 0.1941947191953659, -0.5919250845909119, -0.17374445497989655, 0.7128279805183411, 0.9648765921592712, 0.07643967121839523, 0.4452153146266937, -0.3245781660079956, 1.1405664682388306, 0.030860068276524544, 2.138249158859253, 0.5691801309585571, 0.3617073893547058, -0.9734899401664734, 0.14671680331230164, 0.3127821981906891, 1.7575455904006958, -1.1814115047454834, -1.458978533744812, -0.6121795773506165, -1.218047022819519, 0.8490291833877563, -0.8447483777999878, -0.1250881552696228, -2.462362051010132, -0.03733126446604729, -0.3288620412349701, -0.5519112944602966, -0.10630776733160019, -0.011149836704134941, 0.15866518020629883, 0.8831800818443298, -0.7947943210601807, -0.3484611213207245, 0.41490232944488525, 0.5970727801322937, -0.2601616680622101, -0.02722594141960144, -0.13715237379074097, -0.4431634545326233, -1.0496282577514648, -0.8069037795066833, -1.156029462814331, -1.0583986043930054, 1.7483222484588623, 0.0854923352599144, -0.4595884680747986, 0.3860931992530823, 1.1137166023254395, 0.3542298674583435, 0.10980458557605743, -1.1967754364013672, -1.789770245552063, -0.8805563449859619, -0.4795408248901367, 0.25089120864868164, -0.20117919147014618, 0.37990355491638184, 1.4053595066070557, 0.2201393097639084, -2.01049542427063, -0.8589630126953125, 0.2938709855079651, -1.2844995260238647, 0.37984368205070496, 1.7819000482559204, -1.1619831323623657, -1.376321792602539, 0.040316805243492126, 0.5087411403656006, 0.11236811429262161, -0.390036940574646, -0.7335022687911987, -1.6167032718658447, 1.0619010925292969, 0.6025508046150208, 0.5142442584037781, 2.1886045932769775, 1.7518656253814697, -0.35059574246406555, -0.5151843428611755, -0.7437816262245178, -0.4714778661727905, 0.770600438117981, 0.28227540850639343, -0.44639572501182556, -0.06356723606586456, -0.6338240504264832, 0.3945467174053192, -0.9341883063316345, -0.303560346364975, 0.6979777812957764, -1.4881365299224854, 0.9397454857826233, 0.09533489495515823, -0.5062496066093445, -0.6032795906066895, 0.5051602721214294, 1.0643956661224365, -0.4270784258842468, 1.0299369096755981, -0.7541024088859558, -0.3979724645614624, 0.28481724858283997, -0.5529849529266357, -0.21094365417957306, -0.9514468908309937, 0.21907714009284973, -0.6555957794189453, 1.109792709350586, -0.6087642908096313, 0.16233031451702118, 0.06162886321544647, -0.04600507393479347, 0.8268482089042664, -0.2847736179828644, 0.37312352657318115, 0.13361090421676636, 1.4618302583694458, 0.8313887715339661, 0.7418684363365173, -0.8256296515464783, 0.9564854502677917, 0.8362905383110046, 2.113091230392456, 0.3726106882095337, -0.2693279981613159, 0.09855714440345764, 1.8543050289154053, 0.38036561012268066, -1.2614526748657227, 0.4397960603237152, -0.55239337682724, 0.14945518970489502, 1.0568302869796753, -0.9223284721374512, 0.3986145555973053, -1.651136875152588, -1.176535964012146, -0.4836903512477875, 0.08449544757604599, 0.4372560977935791, 0.21720819175243378, -0.8088042736053467, 0.2501736581325531, -0.9776121377944946, 0.4226544499397278, 1.1974103450775146, -0.1904773712158203, -0.7817005515098572, 0.9316016435623169, -0.2826490104198456, -0.9316165447235107, 0.9798756241798401, 0.014124122448265553, -1.5247350931167603, -0.19881394505500793, 0.973288893699646, 0.6338108777999878, -0.3057905435562134, -1.172104001045227, 1.06251060962677, -0.5871446132659912, 0.13278494775295258, 0.9819119572639465, -0.6860905289649963, 0.10838361829519272, -0.39141637086868286, -0.4395466446876526, 0.4861292243003845, 0.19598296284675598, 0.6333034038543701, -0.1393129676580429, 0.6366092562675476, 1.765173077583313, -0.4199649393558502, 0.9439372420310974, 1.0704400539398193, 2.1624414920806885, 0.9250319600105286, -0.6264094114303589, 2.495797634124756, -1.543503761291504, 0.4641905128955841, 0.08259179443120956, -0.5453946590423584, 1.8362504243850708, -1.6459816694259644, -0.007588605862110853, -0.37063494324684143, -0.48287269473075867, 0.17457252740859985, 0.1591511219739914, 1.1483179330825806, -0.19502253830432892, 1.9980932474136353, -1.0068312883377075, -0.4858590066432953, 0.16908371448516846, 0.5016078948974609, 1.2446694374084473, -0.9991161227226257, -1.0204323530197144, 1.2101941108703613, -0.5515245795249939, -0.7128611207008362, 0.4104767441749573, -1.3826409578323364, 0.12901179492473602, 0.8994672894477844, -1.2454906702041626, 0.6361324787139893, 0.33460670709609985, -1.6814055442810059, 1.7349179983139038, -0.005122185219079256, -0.43249332904815674, 1.184934139251709, 0.036070071160793304, -0.100673146545887, 0.7585578560829163, 0.5257434844970703, 1.89732027053833, -0.09544059634208679, 0.9072255492210388, -0.5258623361587524, -0.7790898084640503, -0.7426760196685791, -0.31058046221733093, -0.2751729190349579, 0.43583622574806213, 0.8795015811920166, 1.2785606384277344, 1.776332974433899, 0.5872207283973694, 1.0021564960479736, -2.490730047225952, 1.2118204832077026, 1.0823074579238892, 1.8656545877456665, -0.08117599040269852, -0.7187076210975647, 0.1598075032234192, 0.03400292992591858, -1.1974326372146606, -1.4162572622299194, -0.65445476770401, -0.29244446754455566, -0.8036397099494934, -0.31345537304878235, -0.18201327323913574, -0.9820435047149658, 0.8208869099617004, 1.3524360656738281, 1.9237948656082153, -0.15603497624397278, -0.025176871567964554, 0.3159050941467285, -0.04882928729057312, -1.7038618326187134, -0.8896991014480591, -0.3752337694168091, -1.5297846794128418, -0.7001504898071289, 0.9923865795135498, 0.06118490546941757, -0.929192841053009, -0.6727638244628906, 0.7293379902839661, 0.7695796489715576, 0.07890079915523529, -0.253162682056427, 1.9410773515701294, -0.04800723120570183, -0.5849695801734924, -0.21404340863227844, -0.22285711765289307, 1.3784922361373901, -1.7279728651046753, 0.6593791246414185, -0.38915854692459106, 0.5980219841003418, -0.8633376955986023, -1.1078683137893677, 0.40509742498397827, 0.8221083879470825, 0.9347432851791382, -1.0415210723876953, 2.5483834743499756, -0.04291810095310211, -0.6851504445075989, -0.8570274114608765, 0.6958681344985962, -0.9351656436920166, -0.09150149673223495, 0.27183476090431213, -0.32697853446006775, 0.7542857527732849, 0.9364666938781738, 1.3570259809494019, 2.2872724533081055, 0.9235275387763977, 0.889120876789093, -1.399224042892456, -0.4027078151702881, 0.5285038352012634, 0.9694877862930298, 1.0415250062942505, 0.5244861245155334, -0.8780905604362488, -0.263681560754776, 0.9223974943161011, -0.1562720537185669, 1.4121659994125366, 2.5585343837738037, -0.9989221692085266, -0.15843386948108673, 0.16762687265872955, 1.9289536476135254, -0.3923206329345703, 1.6788560152053833, -0.35554736852645874, -1.157311201095581, 0.8135288953781128, -0.9090298414230347, -1.3623476028442383, 0.7246513366699219, -0.06902964413166046, -1.2260398864746094, 0.5113897323608398, 0.838855504989624, 0.6063441634178162, -2.330364942550659, -0.8126562237739563, -0.2086005061864853, -1.0039341449737549, -0.14300650358200073, -0.710113525390625, -0.9496601819992065, 1.4107606410980225, 0.6843547224998474, 0.515383780002594, 0.08460313081741333, 0.8824815154075623, 0.5184404253959656, -0.6456750631332397, 1.0201293230056763, 0.28531473875045776, 0.5296895503997803, -0.6182411313056946, 0.3789632022380829, 0.0005585753824561834, -0.3786388635635376, -0.6409283876419067, -0.0027562505565583706, -0.13142387568950653, 1.132176399230957, 1.3152456283569336, -0.1598106026649475, 0.47250932455062866, -0.9911620616912842, -1.4450541734695435, 1.445329189300537, 0.673570990562439, 0.8887066841125488, 0.6302337646484375, -1.3498334884643555, -0.01689010113477707, 0.24071508646011353, -0.05402619391679764, -0.11206517368555069, 0.7093673348426819, 0.5303595066070557, 1.725028395652771, 0.08653143793344498, -1.7036298513412476, 0.24930183589458466, 0.10172142088413239, 1.4971644878387451, 0.27566492557525635, 0.5252642631530762, 0.3722779452800751, -0.295663446187973, 1.1183767318725586, 2.526089906692505, 0.14637435972690582, 0.9190992116928101, 0.29948094487190247, 0.6197876930236816, 1.2703933715820312, 0.7352898120880127, -0.6212790012359619, -1.4463154077529907, -0.531693696975708, -0.8726080656051636, -1.4312461614608765, 0.5245434641838074, 1.8965152502059937, 0.4036051332950592, 0.8914746046066284, 0.4465771019458771, 2.619065999984741, -0.26697662472724915, -0.6235460042953491, 0.4186375141143799, -0.26303648948669434, -0.6827519536018372, -0.8130534291267395, 0.05258653312921524, 1.7177095413208008, 0.40151336789131165, 0.5538201928138733, 0.883319079875946, -1.7477383613586426, -1.7662776708602905, -0.6591533422470093, 1.5633976459503174, 0.24229134619235992, 0.16490887105464935, 1.3194183111190796, -2.425615072250366, 1.0248806476593018, -2.086277484893799, 0.2002599686384201, 0.34703418612480164, -1.4604965448379517, -1.6475422382354736, -0.06794765591621399, 0.08736693114042282, 1.5082505941390991, 0.8244050145149231, 1.2957990169525146, -0.2111722081899643, 0.4154059588909149, -0.008005300536751747, 2.1255884170532227, -0.24306565523147583, 0.35153812170028687, -0.4394291639328003, 1.1867159605026245, 1.6519852876663208, -0.9083241820335388, 1.4248219728469849, 0.9447372555732727, -0.9218776822090149, -0.20833253860473633, 0.6566739678382874, 0.6012507677078247, 1.2311732769012451, -1.4213413000106812, -0.6630210280418396, -1.6269128322601318, -0.2250838726758957, 0.8578022718429565, -0.5700442790985107, -2.5139710903167725, 1.6774568557739258, 0.8182695508003235, -0.5441100597381592, 2.690861463546753, 0.4137902557849884, -0.7481409907341003, 0.5460307002067566, 0.44320276379585266, -0.8499332070350647, -0.33310627937316895, 0.0990850180387497, 0.6714445948600769, -0.37144848704338074, -0.9855691194534302, -0.5170477628707886, -1.0344964265823364, 0.5360620617866516, 0.5085771083831787, -0.6319015026092529, 0.005774670280516148, 0.39465680718421936, -2.4083690643310547, 0.6624292731285095, 1.9598199129104614, 0.0886145681142807, 0.2674998939037323, 0.6636090874671936, -1.2063385248184204, -0.30320343375205994, -0.9089572429656982, -0.5589850544929504, -0.3700329065322876, -0.13985510170459747, 0.031071869656443596, -0.9122547507286072, 1.2268199920654297, -0.00618616072461009, 0.6148240566253662, -1.443865418434143, -1.4787002801895142, 0.43153560161590576, -0.3548729419708252, 0.049706827849149704, 0.11516034603118896, 0.016588907688856125, -0.3049362003803253, 0.9429168701171875, 1.4341624975204468, 0.06315319985151291, -0.04969944804906845, 0.2259923219680786, 0.6263951063156128, -0.3033444583415985, 0.5984207987785339, 1.9177770614624023, 0.4629477262496948, 0.20974332094192505, -0.525461733341217, 1.2136149406433105, 0.28998348116874695, -0.07065078616142273, -0.4965829849243164, 0.4167497456073761, -0.24346625804901123, 0.7120893597602844, 0.019916405901312828, 0.3590884804725647, 0.09002000838518143, 0.5123574137687683, -1.0105810165405273, -0.9884865880012512, 1.0700000524520874, 2.4804391860961914, 0.46272528171539307, -1.8266724348068237, -0.5726737976074219, 1.5281076431274414, -1.2583234310150146, -1.2266358137130737, -0.1772715300321579, -0.6988908052444458, -0.8297670483589172, -0.44199126958847046, -0.2706011235713959, 0.6869156956672668, 0.32737451791763306, -0.5371588468551636, -0.6275941133499146, -0.5549365878105164, -1.4420644044876099, 1.4712110757827759, -0.47506463527679443, -1.0326794385910034, 0.34559497237205505, -0.11022825539112091, 0.6778225302696228, -0.3703916668891907, -0.8391685485839844, -0.6499069929122925, 0.38933902978897095, -0.143528550863266, 0.3412441909313202, 0.07330360263586044, 0.09059257805347443, 0.900445818901062, 0.00027221706113778055, -1.168454647064209, 1.15426766872406, -0.4535004496574402, 0.2942412495613098, -0.6990876197814941, 0.852234959602356, -0.24874532222747803, 0.12086238712072372, 0.818227231502533, 0.3101222515106201, -0.06680401414632797, -1.2161611318588257, -0.4552367627620697, 0.649743914604187, 0.013062475249171257, -1.6841886043548584, 0.8428886532783508, -1.8018901348114014, -3.4170966148376465, 0.1652255356311798, -1.5622525215148926, 0.21384012699127197], [0.9441241025924683, 0.6610409617424011, -0.4630061686038971, -1.060364007949829, 2.205083131790161, 1.742553949356079, -0.4818533957004547, 0.08657969534397125, -0.5250559449195862, 0.35018429160118103, 1.4138072729110718, -0.9373484253883362, 0.5355508327484131, -0.9227433204650879, -0.12462883442640305, -1.4983128309249878, -0.005165609531104565, -0.3095155358314514, 0.15508028864860535, -0.6722803711891174, -0.9301563501358032, -0.007990065030753613, 0.5508464574813843, -0.8036423325538635, -0.36551645398139954, -0.2734542489051819, 0.1587887853384018, -0.4525912404060364, 0.5840319395065308, -0.8782491087913513, 0.5204458236694336, -1.4667764902114868, -1.4803426265716553, 1.6352059841156006, -0.715709388256073, 0.7989773750305176, -1.0596320629119873, 0.18253879249095917, 0.15321339666843414, -0.5837381482124329, 1.125179409980774, -0.16579532623291016, -1.208689570426941, -1.6899569034576416, -0.18892493844032288, 0.4780574440956116, -0.522380530834198, 0.7708061337471008, -1.40788996219635, 0.23936520516872406, 0.16993680596351624, -0.059341710060834885, -2.152313232421875, -0.7302371859550476, 0.8102466464042664, 1.8001834154129028, 1.2067408561706543, -1.6522603034973145, -1.4238922595977783, -0.2511020004749298, 0.7728283405303955, -0.48172441124916077, -0.5207824110984802, 1.0947076082229614, -1.2937155961990356, 1.7554982900619507, -0.7100805044174194, 0.7504830956459045, 0.5034241676330566, 1.6082804203033447, -0.82331383228302, 1.368720293045044, -0.40819215774536133, 0.218194380402565, -0.9924270510673523, 0.6912336945533752, 1.2078865766525269, -0.7240093946456909, 0.7938123941421509, 1.251664638519287, 1.5707905292510986, -2.1323537826538086, 0.682458758354187, 2.5196757316589355, -1.2958956956863403, 0.6745824813842773, -0.5256732702255249, 0.7717075347900391, 0.8216404914855957, -1.3962349891662598, -0.5411012768745422, -0.7936510443687439, -0.7828631401062012, 0.31353652477264404, 0.23344501852989197, 0.5519813895225525, 0.30323323607444763, -1.582047939300537, -1.6913877725601196, 1.4333021640777588, -1.5289098024368286, 1.654173731803894, -0.17257198691368103, 1.155896544456482, 0.5438209176063538, 0.027237193658947945, -1.3582792282104492, 0.6518954634666443, -0.14220742881298065, 0.2939145267009735, 0.3105611205101013, -0.7629351019859314, 0.6611123085021973, -0.9027132987976074, 1.0648874044418335, -0.7598562240600586, -1.169548749923706, -0.2808500826358795, 0.9832438230514526, -0.976062536239624, 0.6538845300674438, 0.03952483460307121, 1.3225107192993164, -1.4814355373382568, -0.1928528994321823, -0.21668419241905212, 1.2902421951293945, -0.4748532772064209, -0.482100248336792, 0.16814856231212616, -0.24695751070976257, 1.6755857467651367, 0.9409753084182739, -0.8693265914916992, 0.1671547293663025, 0.39189574122428894, -1.220301866531372, 1.1042776107788086, 0.38405555486679077, 0.662673830986023, 1.5320544242858887, -0.19916455447673798, 0.5847569704055786, 0.38886547088623047, 0.11719149351119995, 0.7340749502182007, -0.6612033843994141, 0.985058069229126, -1.6940979957580566, -0.503001868724823, 1.764599084854126, 0.7584903240203857, -0.5331348776817322, 1.4208523035049438, 0.4417893588542938, -1.0497201681137085, 1.3009847402572632, -0.04041586443781853, -0.23495616018772125, -1.3504300117492676, -0.9861592054367065, 2.16257381439209, -0.7176579833030701, 1.079226016998291, -2.916313409805298, -0.677150309085846, -0.753288745880127, 0.09469236433506012, 0.9943897724151611, 0.2654639482498169, 0.5081630349159241, -1.5465326309204102, -0.5073184967041016, 1.6403028964996338, -0.6841096878051758, 0.2754516005516052, -0.4172745943069458, -0.5322117209434509, 1.9542406797409058, 0.49574190378189087, 0.2251884788274765, 0.03297809883952141, 0.5854828953742981, 1.8555718660354614, -1.0130282640457153, -1.1619058847427368, 0.6232279539108276, 1.3560657501220703, 0.9481496810913086, -1.8884621858596802, 0.31402915716171265, -0.6064728498458862, -2.1844332218170166, -1.0509428977966309, -2.004760980606079, -1.4886202812194824, -0.3359341621398926, 2.1543822288513184, -1.1526254415512085, 0.0413721427321434, 1.601353645324707, 1.1224411725997925, -1.726689100265503, 1.816670298576355, 1.438610315322876, -0.3554130494594574, 0.8590466380119324, 0.21077856421470642, -2.0870795249938965, 0.7199276685714722, 0.35709884762763977, -1.7557048797607422, -0.6732870936393738, 0.8354353904724121, -1.2225546836853027, -0.5654475092887878, -0.41233402490615845, -2.1729772090911865, -0.7393476963043213, -0.46190908551216125, 1.1122379302978516, 0.3407166004180908, 1.6562330722808838, 0.8300506472587585, 1.2352700233459473, 0.8500219583511353, 0.6129103899002075, 0.8670191764831543, -1.960070252418518, -0.27597668766975403, -0.6940493583679199, -1.3306431770324707, 0.7152095437049866, -0.36699044704437256, 0.2194678634405136, 0.9487040042877197, 1.0748064517974854, -0.5110810995101929, 0.3451855480670929, 0.5832253694534302, 0.31676843762397766, -0.13930533826351166, 1.244699239730835, 0.216083824634552, -0.08672064542770386, -0.9668042063713074, -0.2862739861011505, -1.3056436777114868, 1.670257568359375, -0.7643043994903564, -0.9827737212181091, -0.9405286908149719, 0.5456352233886719, -0.18041621148586273, 1.776471734046936, -1.2109915018081665, 0.4196626842021942, -0.7401612401008606, -0.3686235249042511, 0.6493582129478455, 1.8565336465835571, 1.6960512399673462, -0.017102563753724098, -0.25195544958114624, -0.17360275983810425, -0.5442051887512207, -0.9150677919387817, -0.9049747586250305, 1.1641947031021118, -0.9861461520195007, -0.5760231018066406, 0.7302134037017822, 1.2918444871902466, -0.021632911637425423, -1.3204869031906128, -2.5739095211029053, 0.47261160612106323, 1.152855634689331, -0.4000784754753113, -1.2803517580032349, -0.5055663585662842, 1.0831376314163208, 1.3327678442001343, 1.123116135597229, 0.8064212203025818, -1.1642988920211792, -0.4168131649494171, 0.3040832281112671, 1.1139261722564697, -0.6365782022476196, -0.4879160523414612, -1.3495723009109497, -1.3091051578521729, 0.0459119938313961, -1.4290804862976074, 0.708137035369873, 1.9349637031555176, -1.3349519968032837, -1.6022040843963623, -1.4838818311691284, 2.0670382976531982, 0.724900484085083, -0.24522840976715088, 1.2672783136367798, 1.2633004188537598, 0.24534712731838226, 0.8524090051651001, 0.7383172512054443, 1.0188486576080322, -0.19443446397781372, 2.310831308364868, 0.22239363193511963, 1.5981775522232056, -0.09717586636543274, -1.87728750705719, 0.716094434261322, -0.8446762561798096, -0.7592864632606506, -0.38308799266815186, 1.6173043251037598, 0.35362860560417175, -2.041962146759033, 0.33026185631752014, -1.518107295036316, -0.740150511264801, -1.0730892419815063, -0.06685691326856613, 0.11264771968126297, -0.2065664380788803, -1.1290777921676636, 0.095773845911026, -1.7889511585235596, -0.13502369821071625, -0.23709920048713684, 0.10447636991739273, -0.8772966861724854, -1.0321406126022339, -2.173598289489746, -1.284177541732788, 0.003098804038017988, 0.8018675446510315, -1.4848235845565796, 0.4884732961654663, -1.1734868288040161, -0.1744268536567688, -1.304884433746338, 0.6337278485298157, 0.15312731266021729, 0.44376567006111145, -0.4651675522327423, -0.3603368401527405, 0.6395084857940674, 0.10359970480203629, -0.8717857599258423, 0.19134193658828735, -0.9898678660392761, 1.8348199129104614, -0.7653312087059021, 0.4067412316799164, 2.3021771907806396, 0.13976170122623444, 0.7499213814735413, -0.3446701169013977, -0.11414213478565216, 1.8664759397506714, -0.22043298184871674, 0.5668630003929138, 0.6719672083854675, 0.876282274723053, 2.095618486404419, 0.5079081654548645, -0.6701900959014893, 1.071116328239441, 1.7814019918441772, -1.854109525680542, -0.6564345359802246, -1.3876798152923584, 0.668305516242981, -0.5307068824768066, 1.217589259147644, -0.39255833625793457, 0.27777838706970215, -0.17127972841262817, 0.7137437462806702, -1.816072940826416, 0.42689579725265503, -1.198845624923706, 0.5072765350341797, -0.9709587097167969, 0.47048866748809814, -1.139665961265564, -0.753201425075531, -0.876912534236908, 0.4913283586502075, -1.184558391571045, 0.7235828042030334, 0.16705211997032166, -0.3559105396270752, 1.115605115890503, -0.4530048370361328, -0.13932248950004578, -0.12617340683937073, -0.502988338470459, 1.0738478899002075, -0.854194164276123, 0.8637644648551941, -0.3804434835910797, -0.35174858570098877, 0.3311256766319275, -1.7516673803329468, 0.2868326008319855, -1.2083818912506104, -0.6399706602096558, 0.16643810272216797, -1.1802475452423096, 0.3855973184108734, 0.11698641628026962, 0.31037816405296326, -0.27949637174606323, 2.1287176609039307, -0.6406049728393555, -0.2392539381980896, -0.5527228116989136, -0.026384470984339714, 0.8985787630081177, -0.4812820553779602, 0.0764707550406456, 1.4926342964172363, 0.3448103964328766, 0.37564370036125183, -1.119834303855896, -0.8176723122596741, 0.8889970183372498, 1.4764647483825684, -1.3374648094177246, -0.8071179389953613, -0.7309116721153259, -0.3197028934955597, -0.5285151600837708, 1.934836506843567, 2.7920475006103516, -0.7530983686447144, -0.2519613802433014, 0.46666914224624634, -0.07603005319833755, 0.8424623608589172, 1.9510661363601685, -1.0608243942260742, 0.25895264744758606, -0.9442707300186157, 0.9824786186218262, 0.14660578966140747, 0.1260344386100769, 0.3592771887779236, -0.4911673963069916, 0.24260154366493225, 0.027154289186000824, -1.1211843490600586, 0.4113672077655792, 0.15981271862983704, 0.4926336407661438, 0.9045319557189941, 0.026854010298848152, -1.0908935070037842, -0.11163777858018875, -0.05343146622180939, -0.39409005641937256, -0.167633518576622, 0.6670762896537781, 1.566149353981018, -0.8012687563896179, -1.1556670665740967, 1.311314582824707, -0.8363704085350037, 0.8820582032203674, 0.9534183740615845, -0.20699720084667206, -0.18882843852043152, -1.4927139282226562, 0.4873291850090027, -0.2118704617023468, -0.21933352947235107, -2.5545215606689453, -0.07482627034187317, -0.9839773178100586, -0.4225127398967743, 0.6701220870018005, -1.0706069469451904, 1.0432015657424927, 0.03841922804713249, -0.7039541602134705, -0.11651944369077682, -0.5893422961235046, -0.5872885584831238, -0.5771471858024597, 0.05935632064938545, 0.07772576063871384, -1.0551117658615112, -1.4378087520599365, -0.860467255115509, -1.1436904668807983, 0.5080896019935608, -0.579223096370697, 0.7211396098136902, 0.3752613961696625, -0.6460031867027283, 2.307797431945801, -0.569484293460846, -0.7516142725944519, -0.5301089882850647, 1.1024848222732544, -0.44492173194885254, 0.8964741826057434, -0.05156032741069794, 0.10347142815589905, -0.3792967200279236, -0.13440412282943726, 0.5364369750022888, 0.2619211971759796, -0.4334735572338104, -1.578627347946167, -0.8772823214530945, -0.5033173561096191, 0.26138371229171753, 0.5218521356582642, -0.06995102763175964, 0.03324626758694649, 0.3713834583759308, 0.3516475260257721, 0.8213486671447754, -0.6568911671638489, 0.6134299039840698, 1.2724500894546509, 2.0057168006896973, -0.05398852005600929, -1.382127285003662, 1.3200536966323853, -2.0684778690338135, -1.2868850231170654, -0.44204622507095337, 1.5850580930709839, -1.5659899711608887, 0.42475569248199463, 1.6757724285125732, 1.5889798402786255, -1.4240328073501587, 0.11302106082439423, 0.0031428667716681957, 1.9770535230636597, -1.331035852432251, -0.13366827368736267, -0.18456105887889862, 1.5141890048980713, -0.09264031052589417, 0.6355211138725281, -2.4899799823760986, 1.545494556427002, -0.8751242160797119, -0.8426727056503296, 0.7470409274101257, 0.2539064884185791, 0.5862293839454651, -1.5823423862457275, -1.2695024013519287, -0.7874522805213928, 0.47762778401374817, -0.9206028580665588, 0.20284759998321533, -0.6890848278999329, 0.8700356483459473, -0.8119983077049255, -0.03621925413608551, 0.6047260165214539, 0.6967878937721252, -0.6288259029388428, 0.2896389067173004, 2.370818853378296, -0.375424325466156, -0.7108743786811829, -0.3307463824748993, -0.332307904958725, 0.5090131163597107, -0.26264774799346924, 0.6009269952774048, -0.3755169212818146, -0.3432714641094208, -1.3018689155578613, -2.1252002716064453, 0.9001002907752991, 0.10813546180725098, 0.8223892450332642, -1.461790680885315, -0.8614526391029358, 0.08598709851503372, 0.24040642380714417, -0.7018392086029053, -1.6226474046707153, 1.0774050951004028, 2.3018691539764404, 1.8177525997161865, -0.35652098059654236, -0.34196004271507263, 0.11789416521787643, 0.16084147989749908, 1.06819748878479, 0.714654266834259, -0.6102147102355957, 1.5196064710617065, -1.4410459995269775, -0.12617936730384827, -0.28252026438713074, -0.3929661512374878, -0.0388188436627388, 0.17680932581424713, 0.2792802155017853, 0.20782411098480225, 1.575884461402893, -0.5692334175109863, -2.223662853240967, 0.5194969177246094, 1.4596765041351318, -1.480894923210144, -0.80368572473526, -0.7817755937576294, -0.5351300239562988, 0.8855077028274536, 1.788578748703003, 0.5572646856307983, 1.6150039434432983, 1.1575021743774414, -0.2582591474056244, 1.3421237468719482, 1.3641927242279053, -0.5882498621940613, 0.9437940120697021, 0.6944913268089294, -0.8790711164474487, 0.6288912892341614, 1.6425117254257202, 1.0769939422607422, 0.28446123003959656, 0.15575560927391052, 0.9496424198150635, 1.1153048276901245, -0.27350154519081116, 0.4016115367412567, -0.016307283192873, -0.9084953665733337, 0.16346296668052673, 0.17534512281417847, -0.8560581803321838, -1.583040475845337, -0.5455458760261536, 0.6065060496330261, -0.7550184726715088, -1.8548561334609985, -0.1528284102678299, -0.5688308477401733, -0.2781965732574463, -0.4508090019226074, 0.8474881052970886, 1.838080883026123, 0.24023547768592834, -0.16462303698062897, 1.3136311769485474, 1.3005168437957764, -1.0173289775848389, -2.195673704147339, -0.13672707974910736, -2.3339710235595703, 1.1702567338943481, 1.8757230043411255, 0.2888373136520386, 0.12246697396039963, 1.275456190109253, 0.8721478581428528, 1.497156023979187, 0.9708045721054077, -1.4914705753326416, -0.05229858681559563, 0.64593106508255, 0.5504706501960754, -0.6004346609115601, 0.6400567889213562, -0.027544569224119186, -1.9476373195648193, 0.2034912407398224, -0.08597984910011292, -0.29733991622924805, -0.17998652160167694, 1.728722095489502, 0.844134509563446, 0.036374710500240326, -0.8484508991241455, -0.028014695271849632, 1.5517268180847168, -1.024249792098999, -0.5387521982192993, -1.1707391738891602, 0.8953120708465576, 1.1347993612289429, -0.6994303464889526, -0.07639817893505096, -0.09611517935991287, -0.7874538898468018, 0.19895727932453156, 0.42249003052711487, -0.3082234859466553, -1.3275591135025024, 0.47312846779823303, -0.050672318786382675, -0.47101181745529175, -0.2780722677707672, 1.0228511095046997, 1.4588868618011475, -0.707455575466156, 1.4319171905517578, -0.8950436115264893, -0.03493781387805939, 1.76693856716156, 0.09184998273849487, 2.221764326095581, 0.4252524673938751, -0.40289565920829773, 0.25359660387039185, -1.7664825916290283, 0.26492127776145935, -0.6016301512718201, -0.005663088522851467, 0.962790846824646, -0.02851942926645279, 0.7209429144859314, 0.6115316152572632, -0.5678200125694275, 0.4203712046146393, -1.0567042827606201, -0.4457135498523712, 0.1731305569410324, -0.8459961414337158, 0.3147587776184082, 0.2017032355070114, 0.8088136315345764, 0.8326526284217834, 2.8559443950653076, 0.403708815574646, -0.34681063890457153, -0.3208874464035034, -0.053870849311351776, 0.3179526925086975, 2.302516222000122, -0.23264463245868683, 0.012668727897107601, -0.06978194415569305, -0.907472550868988, 1.3368630409240723, 0.6382220387458801, 0.3208632171154022, 0.8200209140777588, -1.0058753490447998, -0.4241170585155487, -0.7640951871871948, 0.5099371671676636, -0.6347761154174805, 0.6145502924919128, -1.3407955169677734, 0.7096107006072998, 1.1214993000030518, 0.3026566505432129, -0.6873613595962524, -1.3538644313812256, 1.0135587453842163, -0.2752222716808319, 1.2183265686035156, -1.5400264263153076, -2.049252986907959, -1.9751002788543701, -0.4655565917491913, -0.3305099904537201, -0.4934341609477997, 0.8110915422439575, -1.5115326642990112, -0.4950433075428009, -0.4006253778934479, -0.12953542172908783, -0.07849525660276413, -0.13868407905101776, 0.5983825922012329, -1.136260986328125, 0.308450847864151, -0.06049449369311333, 1.9892648458480835, 0.32882827520370483, -0.35091114044189453, 0.13859672844409943, -0.13859061896800995, -0.2012975662946701, -0.08422683924436569, -0.8247705698013306, -0.4583859145641327, 0.29739174246788025, -0.7972460389137268, 0.4058404564857483, -0.7791021466255188, 0.12393742054700851, 1.0033061504364014, 0.13693100214004517, 0.5105327367782593, 1.0830169916152954, 0.3010806739330292, -0.8549928069114685, 1.9214941263198853, 1.330869197845459, -0.7038767337799072, 0.5535679459571838, -1.5169659852981567, 1.1390347480773926, 1.29749596118927, -1.0727167129516602, 0.3279128670692444, -0.5384019613265991, -1.698237657546997, 0.570168673992157, 0.9592329263687134, -0.046116434037685394, -0.7468187212944031, 0.9578658938407898, 0.34087157249450684, 0.34201332926750183, -1.6444501876831055, 1.4166901111602783, 0.27977269887924194, 0.5227503776550293, 1.5468178987503052, -1.2315245866775513, 0.7043688893318176, 1.3266562223434448, 0.7196400165557861, 0.28469929099082947, 2.050144672393799, 0.2692974805831909, -1.2949235439300537, 1.408231496810913, -1.429890513420105, -1.3429573774337769, 0.02637835219502449, -0.9489536881446838, -0.7269003391265869, 0.12025691568851471, -1.5871695280075073, -1.6204670667648315, -0.038373347371816635, 1.0573701858520508, 0.37797194719314575, -0.4974851608276367, -1.3813303709030151, 0.2463379204273224, 0.7088648080825806, 0.09569136798381805, 0.8515039086341858, -0.6394643783569336, -0.07908839732408524, 0.6074405312538147, -0.5481874942779541, 0.5070950388908386, 0.14545482397079468, -1.4831591844558716, -0.09197849035263062, -0.5346125364303589, 0.39710667729377747, -0.5372242331504822, 1.3902785778045654, -1.0233263969421387, 0.07398147135972977, 1.7512365579605103, -1.909683108329773, -1.1877490282058716, 0.8143450617790222, -0.11151058971881866, 1.4673168659210205, 1.5195269584655762, -0.12649881839752197, -0.07367183268070221, 0.7080567479133606, 0.3426409065723419, 0.014403263106942177, 0.5154871344566345, -0.2420690357685089, -0.13979046046733856, -0.9059444069862366, -1.6796772480010986, 1.2211164236068726, -0.2623504102230072, -0.14694976806640625, -1.4908537864685059, -1.4400780200958252, 0.25512930750846863, -0.3668838143348694, -0.9258158206939697, 0.5928629636764526, -2.241337299346924, -0.8731811046600342, -0.6183372139930725, 0.3682304322719574, 0.5682446956634521, 1.4839122295379639, -0.590958833694458, 0.6971744298934937, 0.7859979271888733, -0.1639556735754013, 0.304979145526886, -1.4523981809616089, -0.8812574744224548, 0.769228458404541, -0.20989185571670532, -0.07956917583942413, -0.6014660596847534, -1.1658202409744263, -0.4708402156829834, -0.21931499242782593, -0.319381445646286, -0.43389174342155457, -0.8663287162780762, -0.8625421524047852, -1.708878517150879, 2.5246422290802, 0.0046720560640096664, 0.8819570541381836, -0.30722537636756897, 1.5349652767181396, 0.6750684380531311, -0.14398528635501862, -0.5598893761634827, 0.803788423538208, -0.5771726369857788, -1.901151418685913, 1.7335739135742188, 0.6861553192138672, 0.8416010141372681, 0.6622439026832581, -0.300956130027771, 0.6295122504234314, -0.45528164505958557, -0.11293778568506241, -0.5201002955436707, -0.5230361223220825, 2.0992531776428223, 1.6984986066818237, 0.3923700451850891, 0.5128298401832581, 1.9414260387420654, 0.44232964515686035, -0.6526884436607361, 0.7353175282478333, 0.6592475771903992, -0.131992906332016, -0.4398345947265625, 0.5483618378639221, 0.6289013028144836, -0.30634650588035583, 0.2718704044818878, 0.5472341775894165, -0.8568338751792908, -0.6812028884887695, 0.7590921521186829, 0.9790598750114441, -0.5892506241798401, -0.13027271628379822, 1.1930177211761475, 1.4641022682189941, 1.4323066473007202, -1.0321085453033447, -1.1661365032196045, -1.383088231086731, 1.6723254919052124, 0.9368860125541687, 1.7877683639526367, 1.5286765098571777, -0.6988269090652466, 1.9422303438186646, 0.7516906261444092, 1.418931245803833, 0.7044090032577515, 0.1398104876279831, 0.754746675491333, -1.1617377996444702, 0.02753281220793724, -1.4846280813217163, -0.4079984724521637, 0.5069026947021484, 0.35878822207450867, 0.9197931289672852, -1.1167101860046387, -0.13518552482128143, -0.13096478581428528, 0.8684198260307312, -1.2061927318572998, -0.9240744113922119, -0.872337281703949, 0.15026719868183136, 0.8854735493659973, 0.45655837655067444, 1.6075941324234009, 0.7626157402992249, -1.5934386253356934, -0.3139132857322693, -0.04231105372309685, 0.0634397566318512, 0.26255515217781067, 1.2216999530792236, -0.3311048150062561, 0.7200725078582764, 0.19995370507240295, 0.8746636509895325, 0.07548757642507553, 0.9397396445274353, -0.9881423115730286, -1.3625237941741943, 2.2196905612945557, -1.516865611076355, -0.5709551572799683, -0.579490065574646, 1.1970194578170776, -1.3971381187438965, 1.762904167175293], [1.8882023096084595, -0.02169746346771717, -0.20862136781215668, -1.0390969514846802, 0.4228026568889618, 0.007187201175838709, 1.059758186340332, 0.868503749370575, -1.9993772506713867, 0.4701772630214691, 0.29891353845596313, -1.1055114269256592, -1.6772373914718628, -0.08850998431444168, 0.46405500173568726, 0.9900199174880981, -0.466556578874588, -0.38280627131462097, 0.4366089403629303, 0.16634616255760193, -0.36969995498657227, -0.5748754143714905, -1.8573285341262817, -3.515517473220825, 0.9402981996536255, -0.7247030138969421, 0.49420687556266785, 1.0184203386306763, 1.687973141670227, 0.36386993527412415, 1.9211889505386353, -0.47617948055267334, -0.45492663979530334, 0.26821768283843994, -0.12882302701473236, -0.11827857792377472, -0.7071684002876282, 0.7701840400695801, 0.1309271603822708, 0.779116690158844, 0.6017867922782898, 0.9996585249900818, 0.6376113295555115, -0.2622564136981964, 0.17766447365283966, -0.8215340971946716, 0.7977909445762634, -0.11758691817522049, -0.5533329248428345, 0.5089094042778015, -0.2603181302547455, 0.7183347344398499, -0.007919730618596077, -0.45950958132743835, 0.008140137419104576, -0.6030169129371643, -1.6524806022644043, -1.369694471359253, -0.10540160536766052, -0.6575784087181091, -0.3588067889213562, 0.8210057616233826, 1.9690924882888794, 0.7057779431343079, 0.37447336316108704, 0.1955941915512085, -0.7338137030601501, 0.6524580717086792, 0.7179322838783264, -1.3875607252120972, -0.9934330582618713, -0.944862425327301, 1.6170450448989868, -0.9992073774337769, 0.16156382858753204, -0.5338248610496521, 0.2680697441101074, -0.02949705719947815, 0.6078149676322937, -0.9436322450637817, 0.012689740397036076, -1.4802712202072144, -0.23441223800182343, 0.017177967354655266, 0.795218825340271, 0.7582012414932251, -0.48137104511260986, 0.648819088935852, -0.44061070680618286, 0.31128552556037903, 1.744719386100769, -1.0871721506118774, 1.277174472808838, 0.8695064783096313, 0.18979255855083466, -1.9432005882263184, -0.6646444201469421, -0.6531649827957153, -1.2186756134033203, -0.97629314661026, 0.2281913161277771, -0.29496830701828003, -1.003052830696106, 0.38774630427360535, -0.79544597864151, -0.9760065078735352, 1.7230162620544434, -0.0951959639787674, -1.6224862337112427, 1.225116491317749, 0.9117638468742371, 1.2741315364837646, 0.6413906812667847, -0.08798152953386307, -0.14921458065509796, 0.1146327331662178, -0.28295189142227173, 1.2304723262786865, -0.5088093280792236, -1.6085573434829712, 0.10220766812562943, -0.4762570858001709, 0.3789823651313782, -0.5614083409309387, 1.1165165901184082, -1.77034330368042, 0.7489780783653259, -0.39916616678237915, 0.6027051210403442, -0.09800807386636734, -0.35813647508621216, -0.9930226802825928, -1.6402119398117065, 0.8258049488067627, -1.018966794013977, 0.5359326004981995, 1.2023735046386719, -0.22233137488365173, -0.4517512321472168, -0.1887868344783783, 0.6659735441207886, -0.7022646069526672, -0.3837285339832306, -0.7067796587944031, 0.44538936018943787, 0.5269871354103088, -0.3751503527164459, -0.5038288235664368, 0.3170843720436096, -1.3141428232192993, 0.10423258692026138, 0.47164851427078247, 0.05984584987163544, 0.06724332273006439, 0.8592343926429749, 0.7129640579223633, -2.475336790084839, -0.03277743607759476, 1.7475038766860962, 2.4835774898529053, -0.4866771399974823, 0.07870515435934067, 0.6228829026222229, 1.106663703918457, -2.261882781982422, -0.8314193487167358, 0.16643790900707245, -0.3154110610485077, 1.2245495319366455, -0.3620426654815674, -0.403315931558609, 0.4142283797264099, 0.4408136010169983, 0.13358649611473083, 0.6278830170631409, 0.18874816596508026, -1.2974941730499268, -0.8559557199478149, 1.3939372301101685, 1.046268105506897, -1.1745306253433228, 0.8411024808883667, 0.5554042458534241, -0.26347053050994873, 0.31316444277763367, -0.8223409652709961, 1.0991853475570679, -0.08885969966650009, -1.0982890129089355, -0.9834280610084534, -0.3014456331729889, -0.4757855534553528, 1.1239235401153564, 0.8869999051094055, -0.47830402851104736, 0.7122976779937744, -0.484951376914978, -0.20261918008327484, 0.34485605359077454, 0.2377583533525467, 0.2558915317058563, -0.10968086868524551, 1.1251364946365356, 0.9768227338790894, -1.992754578590393, 1.6387526988983154, 0.7503226399421692, -0.3596857488155365, 0.05391555279493332, 1.065460443496704, 1.3175253868103027, 0.20257319509983063, -0.3261900842189789, 1.9883015155792236, 1.7461344003677368, 1.4539319276809692, 0.46822795271873474, -0.06872895359992981, -2.098834991455078, -0.7701382637023926, -0.7955149412155151, -1.9217464923858643, 0.792887270450592, -0.11819975823163986, 0.8974608182907104, -1.5862654447555542, 0.04862501099705696, -1.7050209045410156, 1.076345682144165, 1.1889411211013794, -1.285304307937622, 0.3623313903808594, 2.314439535140991, 0.021878043189644814, -1.7842496633529663, 0.5215286016464233, -2.2142817974090576, 1.4041640758514404, -0.6221396923065186, -0.7128565311431885, -0.9010934233665466, -0.8456031680107117, -0.07858844846487045, 0.05022752657532692, -1.8372615575790405, 0.703475296497345, -2.144228219985962, 0.43870407342910767, -0.38283097743988037, -0.664031445980072, 0.007736990228295326, -0.6007299423217773, -0.19991084933280945, 0.1335631012916565, -0.22659310698509216, 0.9369597434997559, -0.051487721502780914, -0.3139229714870453, -1.0292510986328125, -0.21992933750152588, -0.4094350337982178, 1.7794684171676636, -1.008221983909607, 0.4336315095424652, 0.3446635603904724, -1.0528982877731323, -0.15153764188289642, 0.0824664980173111, 0.9447579979896545, -1.233115553855896, -0.1675027310848236, -0.6731582283973694, 2.1959972381591797, 1.3139989376068115, 0.20342223346233368, 0.9519320726394653, -0.0027008531615138054, 0.20293337106704712, -0.2901845872402191, -0.5021825432777405, 0.17703522741794586, -1.3088738918304443, -0.3423587679862976, 2.2680561542510986, -1.6157751083374023, 1.2439899444580078, -1.4153085947036743, -0.5398498177528381, 1.303754448890686, -1.2291377782821655, 0.7389617562294006, -0.8889641165733337, 0.9150192141532898, -0.6838452816009521, -0.6437278985977173, -0.4423002600669861, -1.08482027053833, -0.030249111354351044, -2.3895604610443115, 0.9392889142036438, -0.7239169478416443, -2.1973471641540527, -0.3586845099925995, -0.2613525390625, 0.007642198819667101, -0.7315888404846191, 1.2592321634292603, 0.09907060861587524, -0.7204892635345459, 0.42020273208618164, -0.9533811211585999, -0.39894595742225647, -0.4639154374599457, 0.3191080093383789, 0.34905073046684265, -0.9841884970664978, 1.1594423055648804, 0.728168785572052, -0.18616633117198944, 0.4685254991054535, 0.4801739454269409, -0.21233832836151123, -0.20384806394577026, -0.09784546494483948, -1.3898502588272095, 0.2104109227657318, 0.5708451867103577, -0.711950957775116, -0.9713679552078247, -1.4719932079315186, 1.354678750038147, -1.2658987045288086, -0.043912652879953384, -1.327298879623413, -0.05782036483287811, -0.9546967148780823, -1.157220482826233, 1.0249565839767456, -0.016346540302038193, -0.13898520171642303, -0.16767987608909607, -1.4737663269042969, -0.798114001750946, -0.19276534020900726, -0.27893275022506714, 0.12808763980865479, -0.8361175060272217, 1.0743911266326904, -0.353278249502182, 1.4781386852264404, 2.197528123855591, -0.40973183512687683, -0.090082548558712, 0.42121097445487976, 1.5211063623428345, -0.7272761464118958, -1.1027495861053467, -0.9335187673568726, -2.1577436923980713, 0.3654939532279968, 0.27371323108673096, 1.22080659866333, -0.5564323663711548, 1.987890362739563, 0.8056750893592834, -0.9294959902763367, -0.6060911417007446, -0.5386210083961487, 1.6757872104644775, -0.09600161761045456, -0.1369962841272354, -0.14295236766338348, -2.1901235580444336, -0.24475084245204926, -1.1667449474334717, -0.08638528734445572, -0.039452582597732544, 0.05985478684306145, -0.5475831031799316, 1.9954414367675781, -0.5169262290000916, 0.43743208050727844, 0.7227823734283447, -0.5059703588485718, 2.673487424850464, -0.8002115488052368, 0.5753551721572876, 0.5460556745529175, -0.3641493320465088, -0.40389740467071533, -0.8024559617042542, 0.6735676527023315, -0.6463897824287415, 0.8305251598358154, -1.0494014024734497, 0.5051096677780151, 1.483985424041748, -0.5217483043670654, 1.4006437063217163, 0.3459838330745697, 2.047924280166626, -0.9886562824249268, 0.158542662858963, 1.3412387371063232, -0.6365248560905457, -0.265976220369339, -0.175373837351799, -1.4285657405853271, -0.21746346354484558, 0.5132191181182861, 0.7478134036064148, 0.790187418460846, 1.240483283996582, -0.07119296491146088, 0.7944899797439575, 0.8959351181983948, 1.327785611152649, 0.44886547327041626, 0.08915361016988754, 0.08614405989646912, -0.672008752822876, 1.1403920650482178, -0.8638515472412109, 0.9209994673728943, 0.09586171060800552, -1.1000534296035767, -0.17044977843761444, 0.8610307574272156, -0.7214672565460205, 1.8612308502197266, 1.9573949575424194, -1.92532479763031, -0.04771100729703903, -0.1583632081747055, -1.0404194593429565, 0.8445819020271301, 0.8315789699554443, -2.3370230197906494, -1.534011960029602, -0.39442792534828186, -0.13610219955444336, -0.5878890752792358, 0.4731263220310211, 0.7697588801383972, -1.9034322500228882, 0.0817427784204483, -0.22435788810253143, 0.4064626395702362, -1.1020604372024536, 0.6413111090660095, 1.0067342519760132, 1.2408734560012817, -0.4434409439563751, -0.1014510989189148, -1.6500755548477173, 1.6546624898910522, -1.396262288093567, 0.06395411491394043, -0.2136569619178772, -0.3297278881072998, 0.04216017574071884, -1.1214656829833984, 0.4342613220214844, 0.7167754173278809, -0.32425937056541443, 0.6746864318847656, -0.809470534324646, 0.3121834695339203, -0.6941578984260559, 1.0589711666107178, 0.890045166015625, -0.15440471470355988, -0.4958750307559967, 1.7203571796417236, 0.33078235387802124, 1.9735355377197266, 1.7730052471160889, -1.6766256093978882, 0.09038442373275757, 0.45410826802253723, -0.04063895344734192, -1.2171353101730347, -1.2015975713729858, 0.7281002402305603, 0.15757277607917786, -0.06956349313259125, -1.7689281702041626, -2.3415825366973877, 1.0885246992111206, 0.480901837348938, -1.4432460069656372, -0.6880209445953369, -1.4617871046066284, 1.0289840698242188, -0.9311973452568054, 0.0829615369439125, -1.0030369758605957, 0.000741167925298214, -0.04800175875425339, 1.1598947048187256, -0.8326902985572815, 0.7372416853904724, 0.996084988117218, 1.8621207475662231, 1.531137466430664, 0.2266467660665512, 0.14848482608795166, -0.15651366114616394, 0.05236512050032616, 1.531885027885437, 2.749372720718384, -0.5372928977012634, 1.1382290124893188, -0.870573103427887, 0.8462280631065369, 0.3038010001182556, 1.7670307159423828, -1.1242400407791138, 0.5738717317581177, 0.667956531047821, -0.41562893986701965, 0.037912603467702866, -0.6606834530830383, 0.6291263103485107, 0.12355481088161469, -0.03927880525588989, -0.5203220844268799, -0.4616057276725769, -0.7422379851341248, -0.016110621392726898, 0.6959383487701416, 0.7811493277549744, -0.3399180471897125, -0.7924108505249023, -1.4789353609085083, 0.04281279817223549, 1.3897805213928223, -1.2537975311279297, 1.0923117399215698, 0.21339477598667145, -0.0036463544238358736, -0.8157244324684143, -1.266587495803833, 0.4267415404319763, 0.5478626489639282, -1.5580512285232544, -0.9812377691268921, 0.2711138427257538, -0.09089036285877228, 0.9247307181358337, -1.5957231521606445, 1.8619484901428223, 0.2662642300128937, 0.5963786244392395, -1.0920250415802002, 0.8725084066390991, 0.05280373618006706, 1.1824767589569092, 0.21916688978672028, 0.6117827296257019, 0.3869372308254242, -2.1899116039276123, 0.6223384737968445, 0.5542969107627869, -1.0653232336044312, -1.104238748550415, -0.4818451404571533, 1.0232332944869995, -0.19025328755378723, -0.10015838593244553, 1.7057781219482422, 0.9628634452819824, 1.2810992002487183, 0.6925949454307556, -1.7112290859222412, 0.24789540469646454, -1.5741381645202637, -2.1577603816986084, -0.9843083024024963, -1.6485867500305176, 0.5050740242004395, -0.41162556409835815, 1.9483368396759033, 0.3186108469963074, -0.6048877239227295, 0.24731343984603882, -0.9544059038162231, -1.5081309080123901, 1.6822878122329712, -1.097059965133667, 0.8922982215881348, -0.0777268260717392, -0.5439942479133606, -1.1655590534210205, 0.4136732816696167, -0.33051878213882446, -1.3726156949996948, -0.8976178765296936, -0.669981062412262, -0.124332495033741, 0.6554734706878662, -0.20123183727264404, -1.7596464157104492, -0.8567039370536804, 0.8236148953437805, 0.3788186013698578, 0.4932505786418915, -0.8981608748435974, -1.312035322189331, 0.8390424847602844, -1.114933967590332, -0.10939216613769531, 1.9562492370605469, 0.33927297592163086, -0.9878352284431458, -0.9628583788871765, -1.9599839448928833, -1.7366085052490234, 0.04169227555394173, 0.7699927091598511, -0.505383312702179, 0.5543399453163147, -0.40080156922340393, 0.9623619914054871, -1.48755943775177, -0.3783004581928253, 0.7743464708328247, 0.22207409143447876, -1.7642031908035278, -0.895946741104126, 1.5471214056015015, 0.652802586555481, 0.7629284858703613, -0.29569458961486816, -0.8840072154998779, -0.9548695683479309, 1.123732566833496, 1.038138508796692, 1.0235862731933594, -1.3324552774429321, -0.84052973985672, 1.714447259902954, -0.6564385294914246, -1.6691927909851074, 0.09202742576599121, 0.403593510389328, 0.7891032099723816, -0.538891077041626, 1.5535314083099365, -1.0322643518447876, 0.4525269567966461, 0.23496218025684357, -0.34329625964164734, 0.3834143579006195, -0.1455085575580597, 0.673179030418396, 1.0662156343460083, -0.057546548545360565, -1.240899920463562, 0.5496101975440979, -1.5711584091186523, 1.75063955783844, 0.8764492273330688, -0.08379701524972916, 0.6154631972312927, -1.9581743478775024, 0.3504899740219116, 1.3416610956192017, -0.8816938400268555, 0.06838161498308182, 0.2689724564552307, -0.1240164190530777, -0.7033599019050598, 0.07857824862003326, 2.800107479095459, -0.9135822653770447, -0.6267231106758118, -0.75439453125, -2.098593235015869, 0.29484039545059204, 0.03594367578625679, 0.3902038335800171, 1.03713059425354, 0.026423418894410133, 0.14152832329273224, -2.4953999519348145, 1.2960381507873535, -0.29807958006858826, -1.1890063285827637, 1.1976003646850586, -0.4181115925312042, -0.17187976837158203, -0.884355366230011, 0.15448226034641266, -1.6072816848754883, 1.2184454202651978, 1.0158711671829224, -0.9157349467277527, -0.3965567946434021, -0.214105486869812, 0.8176937699317932, -0.9414998292922974, -0.3121131360530853, 1.4238001108169556, -1.895111083984375, -0.8554441928863525, -0.8092437386512756, -1.5074422359466553, -2.0447497367858887, -0.845552384853363, 1.2905470132827759, 3.0188841819763184, 0.461418092250824, -0.7750596404075623, -0.06564763188362122, -1.1915398836135864, -1.4038023948669434, 0.925043523311615, -0.411295622587204, -0.37238529324531555, 1.0200278759002686, 0.534938633441925, 0.9404830932617188, -0.37640443444252014, 0.9148674607276917, 1.2440972328186035, 3.5363805294036865, -0.7662898302078247, -0.3930668830871582, -0.13056567311286926, 0.3167206943035126, 0.9326699376106262, 0.3601952791213989, -2.746762275695801, 0.7541036605834961, -1.6271235942840576, 1.2083849906921387, -0.03245122730731964, -0.5572802424430847, 0.7921915054321289, 1.0253183841705322, -0.5629663467407227, 0.5215153098106384, -2.6289219856262207, -0.4508475065231323, 0.6847070455551147, -0.7407505512237549, 0.2566691040992737, 1.8536885976791382, -0.5740852355957031, -0.9651404619216919, -1.524383544921875, 0.11972735822200775, -0.4571989178657532, -1.6180357933044434, -1.0654852390289307, 0.5477352738380432, -2.335007429122925, -0.5553440451622009, -0.37236732244491577, 0.16381803154945374, 0.10111451894044876, -0.552424967288971, -0.8830575346946716, 1.0867257118225098, -0.629996657371521, 1.1699939966201782, -0.751937985420227, 0.6208893656730652, 0.8204388618469238, 0.7361173629760742, -0.19867567718029022, -1.5884085893630981, 1.0831116437911987, -0.4137706756591797, -0.5491512417793274, 0.40832215547561646, -0.14208607375621796, -1.2254539728164673, 0.6384621858596802, 0.1966208964586258, -1.6248986721038818, -1.0413341522216797, 0.5704659819602966, -0.7919320464134216, 0.2600795328617096, 0.21796537935733795, -0.8522087931632996, 0.22223512828350067, 0.9499818086624146, -1.5305595397949219, -0.015033376403152943, -0.19075249135494232, 1.1323930025100708, 0.11163696646690369, -1.5406075716018677, -1.1800651550292969, 1.5820095539093018, -1.055970311164856, 1.0218520164489746, 0.08755184710025787, 0.22777828574180603, 0.37831947207450867, -0.4358455240726471, -0.33131420612335205, -0.7823768854141235, 1.4261820316314697, -0.6503182053565979, 0.1513596475124359, -1.4697625637054443, 2.0734801292419434, -1.280505895614624, -0.9450057148933411, -0.6452744603157043, -0.6275809407234192, -0.18227621912956238, 0.2154553085565567, 0.596717119216919, -1.3525334596633911, -0.28920289874076843, 0.10149161517620087, -0.4997749924659729, -0.7505216598510742, 0.569126546382904, -1.2136528491973877, 0.21721915900707245, -0.15668022632598877, -0.4793165922164917, -0.5805321931838989, -1.2131757736206055, 1.510057806968689, -0.6674907803535461, 0.2531338930130005, -0.10066714137792587, 1.953679084777832, 0.01802399940788746, 0.4426843523979187, 1.508277177810669, -0.1955111175775528, -1.2145357131958008, -0.01607646979391575, -0.49505919218063354, -1.3303261995315552, 1.5074876546859741, 1.5827758312225342, 0.5428926944732666, -1.5712295770645142, -0.5097135305404663, -0.1352943778038025, 0.9304581880569458, -1.3936710357666016, -1.813962459564209, -0.5370657444000244, -0.01843895949423313, 0.7296931743621826, -0.2387140393257141, 1.323132038116455, 0.46757081151008606, -1.1080682277679443, -1.3944157361984253, 2.3593363761901855, -1.2552851438522339, -0.48808008432388306, -0.2649502456188202, 0.6266986727714539, 1.5949277877807617, 1.152600884437561, -0.5328047275543213, 0.01950773037970066, -0.21717460453510284, -0.6250112652778625, 0.9845608472824097, 0.34663715958595276, -1.2669038772583008, -1.7053006887435913, -0.5451536774635315, -1.1937587261199951, -1.4340029954910278, 1.6322818994522095, 0.06659793108701706, 0.5905883312225342, 1.031718373298645, 0.0006832746439613402, -0.07051654905080795, -2.3508360385894775, -0.21638333797454834, 0.4334173798561096, 0.06127896159887314, 0.5473577380180359, -0.04483684524893761, 0.9829926490783691, 0.3641752302646637, -0.9046428799629211, -0.837209165096283, 0.19717232882976532, 0.8150948286056519, 1.1612348556518555, 0.4286279082298279, 1.1352801322937012, -0.2041187286376953, -1.5272432565689087, -0.7007216811180115, 0.2276323288679123, -1.0807756185531616, -1.7026113271713257, -1.8451424837112427, 0.8875285387039185, -0.36670491099357605, -2.400585889816284, -0.38996776938438416, -1.0290988683700562, -0.30504441261291504, 0.7411210536956787, 0.6053372025489807, -0.4392050802707672, 2.711758852005005, -0.9004654288291931, -0.6213791966438293, -2.065075397491455, -0.545894205570221, -0.10765911638736725, -0.11033080518245697, 0.20913191139698029, -1.1351795196533203, 0.3481689691543579, -1.5106453895568848, 1.7736234664916992, 0.06024997681379318, -1.0582962036132812, -1.2226946353912354, 0.7070396542549133, -0.2796443998813629, -0.8783237338066101, -0.5339630842208862, 1.19219172000885, -1.3695714473724365, -0.24347205460071564, -1.8247973918914795, -0.6250883340835571, 2.0135159492492676, -0.6611082553863525, 0.6358526945114136, 2.5447258949279785, -0.4865970313549042, -0.8219934701919556, -1.5727040767669678, -0.261231392621994, -0.6450555324554443, -0.30838027596473694, 1.8167593479156494, 1.1381657123565674, -0.44615456461906433, 1.0446168184280396, 0.21665218472480774, 1.3844971656799316, 1.1212605237960815, 0.5731366276741028, -0.5110218524932861, -0.018575815483927727, 1.0689244270324707, -0.5486501455307007, -0.9972043037414551, -0.1096821054816246, -0.7661543488502502, -0.21706777811050415, 1.2882970571517944, -1.8451077938079834, -0.3498050570487976, 1.0158783197402954, -0.2747913599014282, 0.7496750950813293, -1.552608609199524, 0.42217060923576355, 0.41796383261680603, -0.13289308547973633, -0.22250261902809143, -0.3689383268356323, 0.3439982831478119, -2.0591976642608643, 2.8013715744018555, -0.027750860899686813, -1.8453572988510132, -1.423353910446167, 1.9017724990844727, -0.9232418537139893, 0.6551486849784851, 0.020347261801362038, 0.37213632464408875, -2.802621603012085, -0.7135226130485535, -0.03772854804992676, -0.8463805913925171, 1.2759860754013062, 1.044319987297058, -1.2177057266235352, 1.0260552167892456, 0.8274695873260498, 1.2562103271484375, -2.2164618968963623, -1.9657903909683228, -0.455530047416687, 0.127630352973938, -2.295870304107666, 0.3008701205253601, -0.7671875953674316, -1.0865994691848755, -0.6212779879570007, -1.4921106100082397, 0.3610193133354187, -0.46038442850112915, -1.9983220100402832, -0.008227293379604816, -0.7288409471511841, -1.074765920639038, -0.044410672038793564, -0.8794342875480652, 0.2714760899543762, -0.5165737867355347, -0.603137195110321, -0.4971574544906616, 1.4266014099121094, 1.3378938436508179, 0.3016337752342224, -0.6022757291793823], [0.7156112790107727, 0.7780645489692688, -1.1177706718444824, -0.5265291333198547, 0.42993032932281494, 1.3197962045669556, -0.4892295300960541, 0.10063070058822632, 0.5677249431610107, -0.2841551899909973, 0.069846011698246, 1.2318284511566162, 0.9339290857315063, 0.38478904962539673, -0.3571709096431732, -0.1726248562335968, -2.2241227626800537, -0.21800145506858826, 0.10620401054620743, -0.021195081993937492, 0.2985713481903076, 0.836740255355835, -1.097154140472412, 1.871345043182373, -1.4848650693893433, -0.3793164789676666, -0.49414825439453125, -0.2171011120080948, 0.3007246255874634, -1.997936725616455, -1.314913034439087, -0.6914982795715332, 0.18109558522701263, -0.6565266251564026, 1.3659312725067139, 0.08778312057256699, -0.7180706262588501, 0.28805655241012573, -0.5050868391990662, -0.2611657381057739, 1.277830719947815, 1.1585261821746826, 0.938274621963501, -1.0068615674972534, -0.06537576019763947, 0.9981337189674377, -0.1293492615222931, 0.5034563541412354, 0.8229931592941284, 1.3790932893753052, 0.6199732422828674, -0.20038147270679474, -0.3060285151004791, 2.6111135482788086, -0.3486236035823822, -1.4543688297271729, -1.2503633499145508, -1.5497615337371826, -0.8930336833000183, -0.11881236732006073, 1.486975073814392, -0.3073316514492035, 0.7334517240524292, -0.7343701124191284, 0.5728386640548706, -0.3136812746524811, -0.15802821516990662, 0.003974266350269318, 0.31725260615348816, 0.8083245754241943, -0.7303743362426758, 1.2434085607528687, 0.8535712957382202, -1.840065598487854, 1.7572441101074219, 0.16115549206733704, 0.33390042185783386, -1.0754629373550415, -0.8601303696632385, 1.37804114818573, 0.17316435277462006, 0.22184531390666962, 0.5168033838272095, 0.5589262247085571, 0.9500550627708435, -0.2987777888774872, 1.121853232383728, -1.1749341487884521, -1.012866735458374, -0.899507462978363, 1.170773983001709, 0.8649126291275024, 0.9964339733123779, -0.5554358959197998, 0.12218716740608215, -0.37988582253456116, -0.513967752456665, 0.32955077290534973, -1.764489769935608, 0.6819787621498108, -0.33155322074890137, 0.1269095242023468, -2.0924997329711914, -1.5180411338806152, -0.38331979513168335, 0.3146466612815857, -0.013620036654174328, -2.531352996826172, 0.6893852353096008, -0.3466198444366455, -0.4691223204135895, 0.3328403830528259, -0.05782945454120636, -0.14361025393009186, 2.005222797393799, -0.9036968350410461, -0.15702325105667114, -0.9474361538887024, -0.7714951038360596, 1.0989432334899902, 1.0190086364746094, 0.7504666447639465, -0.20017234981060028, 1.579978108406067, 1.5311716794967651, 1.2127156257629395, -1.4054667949676514, -0.47839435935020447, -0.5337703824043274, 1.3095189332962036, 2.3787240982055664, -0.9459266066551208, -0.8707038164138794, -1.3132330179214478, -0.5908786058425903, -0.5757216215133667, -0.019915930926799774, 0.9832786917686462, 0.5444914102554321, -0.2771569788455963, -1.1930131912231445, -0.9334440231323242, -1.0235977172851562, 0.30515262484550476, -0.35799404978752136, 1.4307368993759155, 0.549020528793335, 1.9534592628479004, 1.1508044004440308, 0.7582414150238037, 0.7832924127578735, 0.7937266826629639, -1.037352204322815, -0.41496291756629944, -0.18000684678554535, 0.46571221947669983, -1.8229243755340576, -1.4098187685012817, 0.3936246335506439, 0.324293851852417, -0.6425307989120483, -0.3852778673171997, 0.9055463075637817, 0.1595492660999298, 0.062305085361003876, 1.8422573804855347, -0.3450634777545929, 2.91575026512146, -1.035005807876587, -0.0984945297241211, -0.7042813897132874, -1.1288448572158813, 0.25142937898635864, 1.5737863779067993, -0.5046834349632263, 0.6623938083648682, -2.272733688354492, 0.3483116924762726, 0.10886834561824799, -0.314108669757843, 2.936678171157837, -1.6538866758346558, 1.1160967350006104, 0.6193667650222778, -0.020555945113301277, 0.5566605925559998, -0.3270988464355469, 0.02724139764904976, -0.09283552318811417, -0.657440721988678, 0.8880371451377869, 0.49729639291763306, 0.2549329400062561, -0.2016613930463791, -0.656973659992218, -0.6355342864990234, 0.4379338026046753, -1.2161167860031128, -0.6489750146865845, 0.7831889986991882, 0.9270936250686646, -0.09509407728910446, -0.052606575191020966, 1.394907832145691, -2.0038578510284424, -1.091434359550476, -0.4873625338077545, -0.3376266658306122, -0.71855628490448, 0.23883669078350067, 0.20730140805244446, 1.0164968967437744, 1.2870736122131348, 1.5422115325927734, -1.2937231063842773, -0.4912760257720947, -1.8306137323379517, -0.7653035521507263, -0.7338277697563171, -1.4088268280029297, -1.138401985168457, -0.7823136448860168, -0.047425005584955215, 0.7392971515655518, 0.5179271697998047, 0.5627339482307434, 0.24669042229652405, 0.0688241720199585, -0.9122138619422913, -2.267951726913452, 0.2370768040418625, -0.20350158214569092, 2.798243999481201, 0.4455017149448395, -0.32552358508110046, 0.7362370491027832, -0.2889994978904724, 1.5085201263427734, -0.4667157232761383, -0.8888028264045715, 0.08023132383823395, -0.2934001684188843, 0.05295228213071823, 0.32983219623565674, 2.2853033542633057, -1.8613133430480957, -1.856812596321106, 0.6963674426078796, -0.7843417525291443, -1.4059830904006958, 0.29688677191734314, -0.20755910873413086, 2.4295570850372314, 1.913061261177063, 0.04376709833741188, 0.040630824863910675, -0.16354897618293762, -1.1510837078094482, -0.0407426692545414, -0.6643106937408447, 0.8405122756958008, 0.9427075982093811, 1.1072499752044678, -0.02893020212650299, 1.2654401063919067, 0.9397671222686768, 0.6355483531951904, 1.6885885000228882, 0.03536808118224144, 0.7647352814674377, -0.7891349792480469, 0.8055466413497925, -1.836251974105835, -0.1785626858472824, 1.7021312713623047, -2.2643301486968994, -0.3591182231903076, -1.1556376218795776, 0.7377851009368896, 0.7342287302017212, -1.5360510349273682, -1.765042781829834, -0.3561158776283264, 0.9460508823394775, 1.2787091732025146, -0.5936823487281799, -2.0579652786254883, -0.32859480381011963, 0.07431844621896744, -0.39786139130592346, 1.607158899307251, 1.4988141059875488, -1.8969067335128784, -0.9279223680496216, -0.24753357470035553, -1.021945595741272, 1.0410425662994385, 0.3377187252044678, 0.02149905078113079, -0.5306726694107056, 0.2201361209154129, -0.06989715993404388, -0.60539710521698, 0.17969024181365967, -1.7513281106948853, -0.5194454789161682, -0.07575152814388275, -1.1863048076629639, 0.5944163203239441, -0.06612049043178558, 0.7954666614532471, 1.955957293510437, -0.8997547626495361, -0.7709557414054871, 0.38709282875061035, 0.35364899039268494, 2.099457263946533, -0.7539657354354858, 0.3066307306289673, 0.5252410173416138, -1.1971969604492188, 0.2277642786502838, -1.3541804552078247, 0.5536154508590698, 0.6243736147880554, -1.031917691230774, 0.3293391466140747, -0.14113494753837585, -0.7622295618057251, -1.1564433574676514, -0.15584267675876617, 0.6490499973297119, 1.4884909391403198, 0.7803732752799988, -0.9992296695709229, -0.11062802374362946, -0.3675137460231781, 0.6580416560173035, 0.850561261177063, -0.816375195980072, 0.9577764272689819, 1.5195057392120361, 0.007733695674687624, -0.30574384331703186, 0.6988488435745239, 0.31776776909828186, -0.20270730555057526, -0.4188452661037445, 0.4768247902393341, -0.561713695526123, 0.521041750907898, -0.26016294956207275, -0.7421944737434387, 0.4857310354709625, -0.5367686748504639, 0.649410605430603, 0.12043225765228271, 1.189908504486084, -1.1505640745162964, -0.5670688152313232, 1.195695161819458, 2.14935564994812, 0.790142297744751, -0.5869804620742798, -0.8651648163795471, -0.5302980542182922, -0.6014583110809326, -1.468707799911499, 0.24543215334415436, 0.18727006018161774, -1.0760421752929688, 0.30300506949424744, -0.233331099152565, -1.6215317249298096, -0.7104068398475647, -1.5632636547088623, -0.2960888147354126, -0.17796413600444794, 0.6699391603469849, -0.10309702157974243, -0.8604509234428406, -0.5242931246757507, 1.203609585762024, 0.5670342445373535, -0.22801899909973145, 0.13081882894039154, -0.5738477110862732, -1.721156120300293, 0.8141846656799316, -0.197312593460083, -0.4612230956554413, 0.9623662233352661, 0.2487025409936905, -0.35204917192459106, 1.970292091369629, -0.5847983360290527, -0.1563354730606079, 2.665673017501831, 0.08975443989038467, 1.265927791595459, -1.3504154682159424, -1.2989470958709717, -0.7116073966026306, 1.5725209712982178, -0.16184021532535553, 0.09600271284580231, -0.8173137307167053, -0.4901694357395172, -0.8335403203964233, -0.23144258558750153, 0.8390146493911743, -1.52279794216156, 1.776131510734558, 1.0644627809524536, -2.2143852710723877, -0.134513720870018, -0.6063892245292664, -0.08498241007328033, 0.2715109586715698, -1.0632848739624023, -2.032841682434082, 1.2876732349395752, -1.1088473796844482, -0.6658726930618286, 1.8123975992202759, -1.841783881187439, -0.04626338183879852, -0.36464443802833557, 0.41590896248817444, 1.6996029615402222, 0.35568171739578247, -1.5988188982009888, -2.1888110637664795, 0.8383800983428955, -0.016312522813677788, 0.6395404934883118, -0.9560568332672119, 0.4293202757835388, 0.24223969876766205, 1.834335207939148, -1.238696813583374, 0.5076254606246948, -1.2335501909255981, -1.1924941539764404, -0.21843445301055908, 0.9410390257835388, -1.9701461791992188, 0.2943287193775177, 0.1741117388010025, -0.15573039650917053, -2.100146532058716, 1.686199426651001, 0.6225677728652954, 1.8039945363998413, 1.2747594118118286, 1.9277400970458984, 0.16226018965244293, -1.2614020109176636, -1.1943354606628418, -0.46803155541419983, -1.3578031063079834, -0.9198916554450989, -0.08758732676506042, 0.48806414008140564, 0.7692392468452454, 0.6604529023170471, -0.5155091881752014, 1.2269285917282104, -0.62564617395401, -1.5110076665878296, -0.20414753258228302, -0.4348562955856323, 0.18904872238636017, -1.9001182317733765, 0.13970471918582916, 0.10687963664531708, -1.5025681257247925, -1.605383038520813, -0.23626039922237396, -0.9649550318717957, -2.1387102603912354, 2.0128695964813232, -0.6753602027893066, 1.0775601863861084, 0.6948473453521729, -0.29210343956947327, -0.9568420052528381, 1.474055528640747, 0.7635413408279419, -1.2847081422805786, -0.4197142422199249, 2.5713887214660645, 1.2415494918823242, 1.5674480199813843, 0.33085542917251587, -0.21424351632595062, -1.459834098815918, -1.6479930877685547, -0.8629681468009949, -0.8684669733047485, 0.5091882348060608, -1.2986676692962646, 0.06544867902994156, -0.8422063589096069, 1.8161417245864868, 0.4554736614227295, -0.22976413369178772, -0.2692486047744751, 1.799628734588623, 1.1251200437545776, 0.22495271265506744, 1.624433994293213, 0.5778493881225586, -0.45233649015426636, 0.5820242166519165, -0.1311422735452652, -1.159274697303772, -0.1169564500451088, -0.4438948631286621, 0.804955244064331, 0.8349359035491943, 1.3113083839416504, 0.4578125476837158, -0.44625747203826904, -1.6660584211349487, 0.19792819023132324, 0.4897151589393616, -0.5143471956253052, 0.33561110496520996, -1.1618958711624146, -0.5949881076812744, -0.4355802834033966, 1.456505537033081, -0.7319371104240417, 0.8244906067848206, 0.42386335134506226, -0.9087275862693787, 0.005320549011230469, 0.7184871435165405, -0.2524193227291107, 0.4919532537460327, -1.9362704753875732, 2.3302559852600098, 0.18027713894844055, 0.6449888944625854, -1.9992653131484985, 0.7408904433250427, -0.5756411552429199, 0.24360233545303345, 0.08799245953559875, 2.187861919403076, -0.41724610328674316, 0.34606924653053284, 0.8091917037963867, -1.6909908056259155, -0.520915150642395, -0.8085561990737915, -1.8044649362564087, -2.9393258094787598, 1.37326180934906, 1.801416277885437, -1.8303587436676025, -1.2540689706802368, 0.10286924242973328, 0.6340211629867554, 1.4918824434280396, -1.6027075052261353, 0.3930699825286865, 0.7524593472480774, 1.6393182277679443, -0.7190473079681396, -0.19364610314369202, -0.891892671585083, 1.0213055610656738, -1.15756356716156, 0.053782738745212555, 0.8927651643753052, -0.6498759388923645, -0.06839185953140259, 1.3433698415756226, 0.5085684061050415, 0.6319727897644043, -0.1197219267487526, 0.6935153603553772, -0.9205901026725769, -1.06863534450531, 1.2183821201324463, -0.03828931599855423, -0.6786721348762512, -0.2126454859972, -0.6523965001106262, 0.8690606355667114, 1.778463363647461, 0.4775340259075165, 0.7022585272789001, -1.1414204835891724, -1.395282506942749, -0.6626589298248291, -1.318017840385437, -0.6528313159942627, -1.5164250135421753, -0.7965725064277649, -0.6920661926269531, 0.19188328087329865, -0.4447987377643585, -0.2402140349149704, -1.8251073360443115, 1.8281077146530151, 1.4860364198684692, -0.5152449011802673, 0.9443829655647278, -1.201996922492981, 1.9310986995697021, -1.7068263292312622, 0.2551689147949219, 0.4169410467147827, -0.6639738082885742, -0.04998667165637016, -0.08967277407646179, -1.189405083656311, -0.6156129240989685, -0.8976070880889893, -0.22200894355773926, -0.5646085739135742, 0.46468883752822876, -1.0559509992599487, -1.2300456762313843, -1.2020549774169922, 1.2230476140975952, -1.0198345184326172, -0.6694561839103699, -0.45259469747543335, -0.8892098665237427, -2.5534310340881348, -0.5167950391769409, -0.3400321304798126, -0.5978925824165344, 0.21128509938716888, -0.8180574178695679, 0.023472577333450317, 2.1116018295288086, 1.1764494180679321, 0.8626145720481873, 0.004707274492830038, 0.10888276994228363, -0.8101778626441956, 1.7572637796401978, -1.5100936889648438, 0.8545665144920349, -0.27053096890449524, 0.10088885575532913, -0.14150379598140717, -1.6244902610778809, 0.8784988522529602, -0.5195417404174805, -0.26084983348846436, -0.8476344347000122, 1.203112244606018, 0.14173702895641327, 0.3836860656738281, -1.3474717140197754, -0.2171018272638321, -0.12098906189203262, 2.27751088142395, -0.11081842333078384, 1.126017689704895, 0.3615008294582367, -0.5150455236434937, 0.8780805468559265, 0.45717135071754456, 0.1973099559545517, -0.41085296869277954, -0.3318708539009094, 0.5767698287963867, 1.6215038299560547, 1.154959797859192, -0.30773988366127014, -0.24940069019794464, 2.1424355506896973, -0.035842057317495346, 2.239652633666992, -0.8779285550117493, 0.6401156187057495, -0.8568394780158997, 0.767122745513916, -1.1994240283966064, -0.906954824924469, -0.6250832080841064, 1.4281692504882812, 0.27865782380104065, -0.9147281646728516, -0.5355159044265747, 0.7129260897636414, 0.2231878787279129, -1.4964014291763306, -0.6281747221946716, 1.652803659439087, 0.727586567401886, 0.7885461449623108, 0.67604660987854, 1.8626164197921753, -0.9547622203826904, -1.112230896949768, -1.5517135858535767, -0.24257442355155945, -1.4167989492416382, -0.29415392875671387, -0.038179270923137665, -0.1831066906452179, 0.3309798836708069, -0.9348468780517578, 0.11403202265501022, -0.476657509803772, -0.47885680198669434, -0.22436915338039398, 0.17661459743976593, -0.824517548084259, 0.39222097396850586, 0.9092215299606323, 1.1518691778182983, 0.2517299950122833, 0.30176636576652527, -0.1827448159456253, 1.3350980281829834, 0.3742474913597107, -0.6869671940803528, 0.8254304528236389, -0.5998579263687134, 2.728269577026367, -2.480379581451416, -0.3666304349899292, 0.9386741518974304, 0.5448625683784485, 2.462857961654663, 0.10322833806276321, -0.18427196145057678, -0.1749739646911621, 0.2152395248413086, -2.209890365600586, -1.5401191711425781, 0.0068485792726278305, 0.24932868778705597, -0.6298032999038696, 1.238715410232544, 0.5853817462921143, -0.27112728357315063, 1.3289666175842285, -1.5315335988998413, -0.5168203711509705, -0.7828640341758728, 1.6764955520629883, -0.4394625425338745, -1.8104207515716553, 0.2599402666091919, 0.21314626932144165, -0.9299306869506836, -0.3997803032398224, 0.3401104509830475, -0.4151706099510193, 0.9038406014442444, -0.2275044173002243, -1.6409891843795776, -0.7225966453552246, 1.8491425514221191, -0.37326931953430176, 0.16939519345760345, 0.22718612849712372, -0.8109864592552185, -0.07287069410085678, -1.2492846250534058, 0.26351574063301086, -0.3625488877296448, 1.66307532787323, -1.3929275274276733, 0.5498400926589966, 0.054162152111530304, -0.5283079147338867, 0.8019003868103027, -0.3424800932407379, 1.226995587348938, 1.730911135673523, -0.593217670917511, 1.1970115900039673, 0.38937562704086304, 0.14592339098453522, -0.31914883852005005, 0.2086964249610901, 0.013623017817735672, -0.041348230093717575, 0.5746164321899414, 0.2190181165933609, -0.7065461277961731, 0.9926891326904297, 0.5866395235061646, -0.7157700657844543, -1.9975223541259766, -0.9505453109741211, 0.27857527136802673, -0.73537278175354, 0.7263005375862122, 1.1031430959701538, 2.8398032188415527, 0.5201936960220337, 0.0038701477460563183, 1.1525899171829224, 0.902908205986023, 0.9342583417892456, -0.5448524355888367, 1.5165202617645264, 0.299825519323349, 0.9435780048370361, 0.5928874015808105, 0.4064429700374603, -1.4469966888427734, -0.361856073141098, 0.5166768431663513, -0.30999621748924255, 0.11788109689950943, -2.051698684692383, -0.9921677708625793, 1.3806698322296143, -0.20197515189647675, 1.4711169004440308, -1.4671905040740967, 0.5783770680427551, -1.3141522407531738, -0.20941172540187836, -0.7723863124847412, 0.7076476812362671, 1.3507444858551025, -0.754736602306366, -0.7597531676292419, -1.3363064527511597, -1.2726855278015137, -1.6279475688934326, -0.09552761912345886, 0.25016966462135315, 0.07156867533922195, 0.46605920791625977, 0.39131155610084534, -0.24923095107078552, -2.249316453933716, 0.43097639083862305, 1.5734444856643677, -0.7582263946533203, -0.9450951814651489, 0.3727089464664459, 1.2972029447555542, 1.0404272079467773, 1.6327553987503052, 1.0484559535980225, -0.06327985972166061, 0.7315560579299927, -0.7953613996505737, 0.613305926322937, -0.7710500955581665, -1.652703046798706, -0.30189716815948486, 0.7345368266105652, -1.3149253129959106, -0.5763483643531799, -0.43923047184944153, 1.0771507024765015, -0.2783755958080292, 0.09254815429449081, 1.1618990898132324, -2.2465529441833496, 0.6515204906463623, -0.5145542025566101, -0.9336773753166199, 0.9345598816871643, 0.8661623001098633, 0.8658212423324585, -1.1425226926803589, -0.49262699484825134, -0.3865630328655243, 0.12066631764173508, 0.9773529171943665, 0.7064396142959595, -0.06003580242395401, 1.1783443689346313, 1.2788715362548828, -0.7350108027458191, -0.1057269275188446, -0.67851722240448, 0.5403056144714355, 1.6663624048233032, 0.03150649741292, -0.8007189035415649, -1.2081668376922607, -2.095024347305298, -0.7153574228286743, -0.41665807366371155, 0.22545412182807922, 0.37275460362434387, 0.3887031078338623, 0.534646213054657, -1.448566198348999, 2.2355616092681885, 0.6990850567817688, 1.0945713520050049, 1.3249253034591675, -0.10351833701133728, 0.1027454137802124, 0.7703729867935181, 0.07190006226301193, 1.4938747882843018, -0.6623855829238892, 0.3052229583263397, 0.5941965579986572, -1.738879680633545, -0.9249391555786133, -0.0939607173204422, 1.1902410984039307, 0.21419218182563782, 1.4392645359039307, -0.06608760356903076, 0.27071988582611084, 2.269341230392456, 0.1683771312236786, -1.4835633039474487, -0.4088049829006195, -0.0746796727180481, 0.6617051362991333, -1.5046825408935547, -0.22433073818683624, -0.07608262449502945, -0.29292866587638855, 0.3768719434738159, -1.2077956199645996, 1.039372444152832, 1.0465643405914307, 0.9736286997795105, -0.020339885726571083, -1.7107073068618774, 0.9706538915634155, -2.1770498752593994, 0.3684294521808624, -1.894081950187683, 1.290875792503357, -0.31191539764404297, 0.734035074710846, -0.6452949047088623, 0.03579321503639221, -1.1300833225250244, -0.3604121506214142, -0.4628460109233856, 0.043048858642578125, 0.7086780667304993, 1.164725422859192, -1.2569222450256348, -0.5125321745872498, -0.02384817972779274, 0.5905687212944031, -0.388731986284256, -1.5609091520309448, -0.7451161742210388, 0.45556873083114624, 0.9842420816421509, 0.3845635652542114, -0.79361492395401, -1.4508312940597534, -1.546747088432312, 0.7716572284698486, 0.2176431268453598, 1.0146416425704956, 1.773026943206787, 0.15540282428264618, 0.5652158260345459, -1.012959361076355, -0.3899524509906769, 1.1731643676757812, -0.4113563895225525, 0.5189316272735596, 0.05932381749153137, 0.1207127496600151, 1.045898199081421, -0.2704930901527405, 0.23678773641586304, 0.14369767904281616, 0.4128758907318115, 0.4259806275367737, 0.49399763345718384, -1.4516322612762451, 0.9191661477088928, 1.7258009910583496, 0.5530019998550415, 0.14147047698497772, 0.5034052133560181, 0.8208138942718506, 0.5552791357040405, -1.468907117843628, 0.8497346043586731, -1.1019766330718994, 0.6411983966827393, 0.5564461350440979, 0.4179271161556244, -1.2243870496749878, 1.8022706508636475, -0.5416032671928406, 0.5107656717300415, -0.6342325210571289, -1.202397108078003, -0.2834513485431671, 1.1646134853363037, 1.9471852779388428, -0.4417939782142639, -1.0674924850463867, -0.47971194982528687, -0.7120980620384216, -1.5640133619308472, 2.3414182662963867, -0.2643280625343323, -0.346665620803833, 0.03346417471766472, 1.7995094060897827, -1.1379351615905762, -1.3578131198883057, 0.028482750058174133, 1.52605402469635, 1.91214919090271, -0.33371931314468384, -1.2919085025787354, -0.14749321341514587], [0.8242406845092773, 0.7857686281204224, -0.5976405143737793, -0.16250663995742798, 1.1017961502075195, 0.6282802820205688, 0.1886831820011139, -0.22264918684959412, -0.7426783442497253, 0.48462170362472534, -0.5462344884872437, -0.6302555203437805, 0.328706830739975, 0.7056391835212708, -0.6295373439788818, 0.7237609624862671, -0.010621964000165462, 1.1879085302352905, -0.6402367949485779, -0.17771777510643005, 0.7209157347679138, 0.3008994460105896, -0.2710494101047516, -0.13509684801101685, -0.8036693930625916, -3.3960483074188232, 1.1164839267730713, -1.213945984840393, 0.13141050934791565, -0.3285645544528961, 0.7634500861167908, -1.3557738065719604, 1.5829817056655884, -1.1632150411605835, 0.33919379115104675, -2.9173648357391357, -0.14971648156642914, -1.653190016746521, 1.1497292518615723, 1.2921217679977417, -1.2536730766296387, 0.8526241779327393, 0.9017450213432312, -0.6862241625785828, 0.2213742733001709, -1.7487146854400635, 1.0338971614837646, -0.43002045154571533, -2.3112568855285645, 0.6816264390945435, 1.2632554769515991, 1.0795702934265137, 0.8730862736701965, -1.0065388679504395, 1.3677085638046265, 0.8530091047286987, -0.4396418035030365, 0.2215588390827179, -1.0914664268493652, 0.35317209362983704, 0.8732417225837708, 1.3018797636032104, -1.5556560754776, -0.20521201193332672, 1.002449870109558, -0.1681717187166214, 0.1472708135843277, 0.6033840775489807, 1.101578950881958, 0.31755879521369934, -0.12396964430809021, 2.1055402755737305, 0.6221932172775269, -0.7876501083374023, 0.9814691543579102, 1.6004276275634766, -0.9227606654167175, -0.48218169808387756, -0.17982695996761322, -1.320294976234436, -0.04691217839717865, 0.9608837366104126, 0.48256799578666687, -0.5202907919883728, -0.34036755561828613, 0.7718758583068848, 0.12438458949327469, 0.7335077524185181, 0.4212096333503723, -2.5438172817230225, -0.5225121974945068, 0.5279881358146667, 1.8653148412704468, 1.7545137405395508, -0.33808010816574097, 1.2798067331314087, 2.297569751739502, 0.3740687966346741, -0.6337842345237732, 0.6444142460823059, 0.7420144081115723, -1.2071559429168701, 0.12987886369228363, -1.0331945419311523, -0.1000654324889183, -0.6959384083747864, -1.4097297191619873, -1.7407312393188477, 0.4614197611808777, 0.668889582157135, 0.8128824830055237, -0.23564933240413666, 0.5617572069168091, 1.815477728843689, -0.14001618325710297, 0.20715084671974182, 2.6429831981658936, -0.16559164226055145, 2.315192461013794, -1.1764450073242188, -0.2550796866416931, -1.0413323640823364, -0.7062322497367859, 0.8129854798316956, 0.6464472413063049, 0.7539032101631165, -2.006706714630127, -0.48963719606399536, -1.0718145370483398, -0.8034610748291016, -0.6371403336524963, -0.8063946962356567, 0.0019929741974920034, 0.3748241364955902, -0.3461117148399353, -0.10565902292728424, -0.12019263207912445, 1.1132491827011108, -0.017012692987918854, -0.6832435131072998, -3.8901331424713135, -1.0734539031982422, -0.06581957638263702, -1.0126501321792603, -1.2146741151809692, 0.1229410171508789, 0.0005757176550105214, 0.436993271112442, 0.15084384381771088, 0.6749484539031982, -0.6392128467559814, 1.2056025266647339, 0.5707064867019653, 1.30398690700531, 0.4273883104324341, -1.5021270513534546, 0.2927989363670349, 0.914966344833374, 0.13467203080654144, 0.4948042631149292, 0.6005029082298279, -0.7047851085662842, 2.9541046619415283, -1.1920552253723145, -0.5530189275741577, 1.2992514371871948, 0.049149107187986374, -0.47539031505584717, -0.47289952635765076, -1.068142294883728, 0.3738102614879608, -1.011764407157898, -0.5851274728775024, -2.8188064098358154, -1.3821511268615723, 0.9414149522781372, -1.3930866718292236, -1.1086560487747192, -1.1306005716323853, 0.4332405924797058, 0.6715898513793945, -0.02046697586774826, 0.17472133040428162, -0.7320932745933533, -0.9627087712287903, 0.3790535628795624, -0.746497631072998, 1.22636079788208, 1.6842823028564453, -0.12074172496795654, -0.04874604195356369, 1.0360383987426758, -1.2562941312789917, 0.610440194606781, -0.38363268971443176, 0.9616174697875977, 0.8376035094261169, 0.5035788416862488, 0.34108009934425354, 0.03073457069694996, -0.39046287536621094, 1.1751518249511719, -0.47720426321029663, 1.584346055984497, 0.3045007288455963, 0.1629399210214615, 0.01743835210800171, 1.178165316581726, 0.13992483913898468, 0.14757215976715088, -0.3089875280857086, 0.5596805214881897, -0.6371865272521973, 1.3504726886749268, -0.7424799203872681, -0.21509137749671936, 1.1001712083816528, 0.47343602776527405, 0.7227487564086914, -1.1727063655853271, -0.5993542671203613, 0.5609971284866333, 0.024633269757032394, 0.4965549111366272, -1.2235158681869507, 0.4428291916847229, 0.533610463142395, -0.030689643695950508, 1.7741619348526, -0.3253766894340515, -0.17910324037075043, 0.4607488512992859, -1.115257978439331, -0.23976050317287445, 0.5052648782730103, 1.3336931467056274, 2.1363463401794434, -0.5069262981414795, 0.8843388557434082, -1.3614778518676758, 0.18802642822265625, -1.4765294790267944, -0.9890720248222351, 0.2312922477722168, -1.1328678131103516, -0.3676344156265259, -0.08198465406894684, -1.2899545431137085, -0.2777688801288605, 0.31175702810287476, -0.24756266176700592, 0.12507709860801697, -0.6244505047798157, 0.34521597623825073, 0.7683869004249573, -1.158271312713623, -0.9711306691169739, 0.8108851909637451, -0.7327713966369629, -2.2141263484954834, -0.35408973693847656, -0.23640474677085876, -0.9425207376480103, 0.0390930213034153, 0.11021540313959122, -0.31967228651046753, 0.5639278888702393, 1.9302475452423096, -0.589887261390686, -0.09382570534944534, 0.21883711218833923, 1.404089331626892, -0.9342387914657593, -0.8645780682563782, 0.9522338509559631, -0.945616602897644, -0.7178019881248474, 0.6709051132202148, 1.1201547384262085, -0.8756675720214844, -0.5301810503005981, 1.366692066192627, 0.6901101469993591, -0.8247085213661194, 1.0884151458740234, -1.2393089532852173, -2.346151828765869, -1.011053442955017, 0.21244122087955475, 0.16911061108112335, 1.6382302045822144, -2.5240039825439453, 1.0006544589996338, 1.2288557291030884, 0.1338604837656021, 0.9900813102722168, -0.5239049792289734, 0.2647329866886139, 1.030534267425537, -0.6230974793434143, -1.0586193799972534, 0.3630717992782593, 1.2782557010650635, -0.07109648734331131, 1.6965306997299194, -0.03331110626459122, 0.44370123744010925, -0.7288925051689148, 1.380518913269043, -0.08748839795589447, -1.0001933574676514, -2.1663098335266113, -0.06653840839862823, 0.6202338933944702, 0.8186447620391846, -0.8661752939224243, -0.48485860228538513, -0.004413388203829527, -0.6008263230323792, -0.2005784660577774, -0.29701414704322815, -0.6593779921531677, -0.7753593325614929, -0.47296053171157837, -0.03234512358903885, 1.5290076732635498, -0.8468948006629944, 0.20928539335727692, 0.7025245428085327, -0.1829926073551178, 0.9384183287620544, 1.239377498626709, 1.5354962348937988, 1.3343175649642944, 0.6959302425384521, 0.5291285514831543, 0.4752880930900574, -0.24314606189727783, 0.671566367149353, -0.8343161344528198, 0.6853365898132324, -0.2621792256832123, 0.5520066618919373, -1.0381481647491455, -0.47499364614486694, -0.22630327939987183, -0.7833778262138367, 1.141782283782959, -1.2638108730316162, 0.9057496786117554, -0.12732693552970886, -0.2989885210990906, -1.6956007480621338, -2.055205821990967, 0.4791812002658844, 0.3728519678115845, -0.21837440133094788, 1.473056674003601, -0.4968387186527252, 0.5615018010139465, -1.251798391342163, -0.04638835787773132, 0.4074176847934723, -1.9104971885681152, -0.39098426699638367, -2.1074025630950928, -0.0008481702534481883, -1.3389819860458374, -2.0583200454711914, -1.2541831731796265, -1.098053216934204, 0.67767333984375, -0.024092888459563255, -0.7072403430938721, -0.21864211559295654, -0.2351965308189392, 0.48755577206611633, -0.7719940543174744, -0.7682927250862122, 0.5212267637252808, 0.21795986592769623, 0.9243979454040527, -0.5401293039321899, 0.9978480339050293, 2.2620081901550293, -0.46361464262008667, -1.2575218677520752, 1.0421191453933716, 0.5789927840232849, -0.9457822442054749, 0.4269946217536926, 0.4199136793613434, -0.4152474105358124, 0.6540780663490295, -0.09258026629686356, -1.1788368225097656, 0.36117658019065857, 0.7027360200881958, -2.2790420055389404, 0.8652191758155823, 1.033992052078247, 1.5089391469955444, -0.36069753766059875, -0.1369403600692749, -0.5194225907325745, 0.9439113736152649, 1.1054129600524902, -1.3985931873321533, 0.13272616267204285, 0.35466963052749634, 0.41213297843933105, 0.4268436133861542, 0.7838247418403625, -0.2225838303565979, 0.6765680313110352, -0.7294257283210754, -1.1354374885559082, 0.1652759462594986, 1.312543272972107, -1.2106397151947021, -0.4335724711418152, 0.41540825366973877, 0.16870595514774323, -0.08080624788999557, 1.6974866390228271, 0.13627417385578156, 0.8328932523727417, 0.9479528665542603, -0.34660032391548157, -0.35876426100730896, -0.9968669414520264, 0.745725691318512, 0.8112776279449463, 0.06586621701717377, -0.8908689022064209, 0.03700801357626915, 1.2847404479980469, 1.0791019201278687, 0.48137205839157104, -2.380324125289917, 1.6589597463607788, -1.2618539333343506, 0.6898869276046753, 1.1831281185150146, -0.3512437641620636, 1.2401260137557983, -1.1713755130767822, -0.3196028470993042, -0.5952164530754089, 0.9814001321792603, -1.0392638444900513, 0.07126349955797195, 0.2935572564601898, 0.3877793550491333, -0.35908079147338867, -0.4898032546043396, 0.10806292295455933, 0.5176047086715698, 0.5214172601699829, 0.7009680271148682, -0.9204789996147156, -0.22316978871822357, 0.6614534854888916, 0.9255238175392151, 0.16661880910396576, 1.606724739074707, 0.622502326965332, -1.2277425527572632, 0.2589254379272461, 1.8377840518951416, -0.8746930956840515, -1.1575075387954712, 0.2050255835056305, -0.0922064334154129, -0.7238049507141113, -0.6551897525787354, -0.6712994575500488, 0.5206664204597473, 0.9509329795837402, -0.10320904105901718, 1.4899797439575195, -0.3530668318271637, 0.29376021027565, -1.166667103767395, -0.31733831763267517, 0.8467563986778259, -1.942755103111267, 0.09945065528154373, 0.21395517885684967, 0.2401343137025833, -0.49595850706100464, -1.5494464635849, 0.6917321085929871, 0.07318644970655441, -0.16210989654064178, 0.22003187239170074, -0.18874414265155792, 1.1755576133728027, 3.012812852859497, 1.1683688163757324, -0.36926063895225525, -0.6948249936103821, 0.7940549850463867, 0.9348650574684143, -0.12142638862133026, -0.5101430416107178, -0.09332138299942017, -0.383492648601532, 0.5143917798995972, -0.13749746978282928, 0.4714866876602173, 0.41695350408554077, -0.11561207473278046, 1.8100976943969727, 0.8604481220245361, -1.176356554031372, 0.14007392525672913, 0.9370100498199463, 1.3813896179199219, 1.0293604135513306, -0.19968166947364807, 0.2592194974422455, -0.36010006070137024, -1.8043299913406372, -0.6518163681030273, -0.4256095886230469, 0.49255990982055664, -0.2646523714065552, 1.6241917610168457, -0.8639313578605652, -1.3522788286209106, -1.0862318277359009, 1.1728489398956299, -0.5600190758705139, 0.041815999895334244, -0.6788358688354492, -0.6080864667892456, 2.4329798221588135, 1.439921498298645, -0.9794316291809082, -1.7185392379760742, -1.0861414670944214, -1.168246865272522, -0.012667665258049965, -1.6396957635879517, 1.5928386449813843, -1.1392474174499512, -1.5098685026168823, -0.648406445980072, 0.3597624897956848, -0.448703795671463, 1.792937994003296, 0.8578017950057983, -0.25418561697006226, -0.8744494915008545, 0.7926475405693054, 0.8056485056877136, 0.1744450032711029, -0.6921266317367554, 0.8952873945236206, 0.11381587386131287, 0.8800356388092041, -2.285646438598633, -0.246510848402977, 0.6609935760498047, 0.7599753141403198, 0.3236137628555298, 1.383197546005249, 0.5401667356491089, 0.1030869260430336, -1.2538213729858398, 0.9419747591018677, 0.6316123604774475, -0.7154425978660583, -0.4416247010231018, 0.7409639954566956, 0.6729613542556763, -0.503599226474762, -1.11769437789917, -1.61214280128479, -0.46467024087905884, -1.8017569780349731, -2.0636866092681885, 0.45145684480667114, 1.0196517705917358, 0.4972739815711975, -0.9001089334487915, -0.550502598285675, 0.8686989545822144, 0.37140223383903503, -0.5514093637466431, -0.19260148704051971, -1.6275572776794434, 0.6065162420272827, 0.33393311500549316, 1.8835113048553467, -0.052646540105342865, 2.3204147815704346, -1.3855984210968018, -0.18870976567268372, -0.28962233662605286, -1.0154370069503784, 0.7841039299964905, -1.0707650184631348, 1.921572208404541, -0.5729813575744629, 1.0094367265701294, 1.2987186908721924, -1.8460701704025269, 0.9242867827415466, 0.695393979549408, 0.1693968027830124, 0.7062236666679382, 0.2953146696090698, 0.38849711418151855, 1.4254577159881592, -1.5563256740570068, -1.2811853885650635, 1.257732629776001, 0.31526970863342285, -0.3946976065635681, -2.0035200119018555, 1.069681167602539, -0.464397668838501, -1.6508266925811768, -1.4411165714263916, -0.2578544020652771, -0.2525225579738617, 0.5320517420768738, -0.510385274887085, 0.49779820442199707, -0.44533103704452515, -0.16951903700828552, 2.0433502197265625, 0.4200803339481354, 1.540737271308899, 0.6639916896820068, 1.0446816682815552, 1.1609500646591187, 1.1084781885147095, 0.7529299259185791, -0.37848812341690063, 0.45635542273521423, -0.19308391213417053, -1.746925950050354, -1.3597054481506348, 0.2026069611310959, 0.7319068312644958, -0.8901275396347046, 0.5118058919906616, 0.16123849153518677, -0.24362654983997345, 1.5292237997055054, -1.3829630613327026, -0.87999027967453, -0.17351342737674713, 0.048214104026556015, -0.7347458004951477, -1.131200909614563, 1.5702673196792603, -1.2757108211517334, -0.35223492980003357, -0.2540445625782013, 0.5764088034629822, 1.266530990600586, -0.185252383351326, -1.0240323543548584, 0.01822710782289505, 0.7854573130607605, 0.9589208960533142, -0.3913406729698181, -1.184254765510559, 0.7449447512626648, -0.7463365793228149, -2.063256025314331, -1.2380092144012451, 0.9115861654281616, 0.0571090430021286, 2.1773250102996826, -1.3919367790222168, -0.5878547430038452, 0.8340174555778503, -0.2059316784143448, -0.25538012385368347, 1.6892859935760498, -0.6435105800628662, -1.2712072134017944, 2.1956961154937744, 1.3727198839187622, -0.29133906960487366, 0.30081886053085327, 1.457198143005371, 0.5584741234779358, -0.18296939134597778, 1.511594533920288, 0.2655196189880371, 2.0013535022735596, 0.44847550988197327, 1.3907917737960815, -1.4261850118637085, -0.6975111365318298, -1.327845573425293, 0.1601473093032837, -0.4685264527797699, -0.638357400894165, 1.1719753742218018, 0.5419874787330627, -0.33053314685821533, 0.8736532330513, 1.4624476432800293, -1.0640519857406616, 0.7345073819160461, -0.5443294048309326, -1.2417742013931274, -1.349354863166809, 0.7031185626983643, -0.379709929227829, 0.6857967972755432, 1.2958874702453613, -0.7335337400436401, -0.11564037203788757, -1.0303584337234497, 0.17502199113368988, -2.268551826477051, 1.4864829778671265, -0.7285468578338623, -0.21140065789222717, -0.30470797419548035, 0.06395043432712555, -1.0619879961013794, -1.3227875232696533, 1.7641736268997192, 0.266875296831131, -0.5506040453910828, 0.11565174162387848, 1.287824034690857, -0.7867857217788696, 1.3336976766586304, -0.9635272026062012, 0.7807406783103943, 2.8946051597595215, 1.4155365228652954, 0.14417868852615356, 0.3071754574775696, -0.1981317698955536, -0.24157224595546722, -0.07716444134712219, -0.3091370463371277, -1.8361330032348633, -0.2579621970653534, -1.8054238557815552, -1.2075434923171997, 0.4365662932395935, -1.0991694927215576, -0.0020600149873644114, -0.5863915681838989, -0.4608871340751648, -0.08890774101018906, 0.7008503079414368, 1.1757434606552124, 0.6400872468948364, 1.734817624092102, -0.22741162776947021, 0.02797694504261017, -0.698136568069458, 2.400637626647949, 0.6831501126289368, -0.45657896995544434, -0.5292227864265442, 0.3510960340499878, 0.2679692506790161, -1.0983915328979492, 0.361613929271698, 0.7021512389183044, -0.16257283091545105, -1.849301815032959, 0.3428571820259094, 0.47550874948501587, 0.33215251564979553, -1.9090322256088257, -1.3773934841156006, 0.2184128314256668, -0.8594052195549011, 1.9582966566085815, -0.0043400926515460014, 0.27176782488822937, -0.2066899687051773, 0.7965335249900818, 0.23028528690338135, 0.9286600351333618, -1.2215673923492432, 0.5610871315002441, 0.4654870927333832, -0.1612011194229126, -1.7653213739395142, -0.6029816269874573, -1.5751588344573975, 0.8936696648597717, 1.5313822031021118, -0.8244910836219788, -0.042806077748537064, 0.7106550335884094, 0.1315482258796692, -0.9318149089813232, 0.5074397325515747, -0.16357548534870148, 0.5733270645141602, 0.9428287744522095, -1.0973024368286133, -0.4404829144477844, -0.62493896484375, 0.6909077167510986, -0.09603820741176605, 0.32105395197868347, 0.02093849889934063, 1.9682908058166504, -0.8958781957626343, -1.0356703996658325, -0.28852665424346924, 0.9893693923950195, 1.287920594215393, 0.6763882040977478, -0.33144497871398926, 0.7777767777442932, 3.09381365776062, 0.36415496468544006, -0.36653244495391846, 0.23846249282360077, 0.08554103225469589, -0.6283197999000549, -0.812942385673523, 1.0988017320632935, -1.2815370559692383, 0.41841205954551697, 0.32806333899497986, 0.3943719267845154, 0.8139322996139526, 0.5467294454574585, -0.5768661499023438, 1.6769789457321167, 0.0778203010559082, 1.612380027770996, 0.630454421043396, 1.1520757675170898, -0.8593223094940186, 1.0534862279891968, 0.8010416626930237, 0.20339469611644745, -0.5571563243865967, -0.061987604945898056, 0.5300162434577942, 2.1233363151550293, -0.3388255536556244, -0.09503314644098282, -0.758573591709137, -0.02930716797709465, -0.14555449783802032, -0.4387345314025879, 1.4748499393463135, 1.4141664505004883, -0.4614664316177368, 0.8794260025024414, -1.32563316822052, 0.31350600719451904, 0.8618764877319336, 1.3457667827606201, 0.6182025074958801, -1.6918449401855469, -1.417850375175476, 1.56479811668396, -0.11054419726133347, 0.4323340654373169, 1.8462450504302979, 0.9365878701210022, 0.3720706105232239, -1.267238736152649, -1.9907751083374023, 0.6164454221725464, -0.598859429359436, 0.6398113965988159, 0.3185594975948334, 0.0665844976902008, -0.6151453852653503, -0.6295779943466187, 0.6602008938789368, -0.8947683572769165, 0.028591174632310867, -1.1022175550460815, -0.06618043035268784, -1.2235589027404785, -0.09192628413438797, 0.10285555571317673, -0.11946320533752441, -0.21273154020309448, 0.09963315725326538, -0.5638939738273621, -2.2182223796844482, -0.2020237296819687, 1.1256803274154663, -0.241457000374794, 1.5735764503479004, -0.7961771488189697, 0.1334935575723648, 0.603587806224823, 0.7663570642471313, 0.33036452531814575, 0.5980396866798401, -0.022643331438302994, -0.37175798416137695, 0.08846282958984375, -1.0952590703964233, 0.36919841170310974, 0.018176961690187454, -0.5104655623435974, -1.4737178087234497, -3.0469393730163574, -1.129246711730957, 0.9504289627075195, 1.2014180421829224, 0.17233148217201233, -1.4266924858093262, 0.16811448335647583, 0.8927167654037476, 0.23771928250789642, 0.7034791707992554, -0.40388593077659607, -0.5664038062095642, 0.4529930353164673, 0.611031174659729, -0.6124461889266968, -1.2483773231506348, -0.5739893913269043, -0.6636548042297363, 0.7049537897109985, -0.24944469332695007, -0.6152910590171814, -0.5475417971611023, -0.22433136403560638, -1.841753602027893, 0.28136682510375977, -1.0834909677505493, -1.808195948600769, 2.637871026992798, -1.2807443141937256, 0.8144736289978027, 0.014750816859304905, 0.06570590287446976, 1.649048924446106, 0.8708170652389526, -1.4173611402511597, 0.173786461353302, 0.35740748047828674, 0.8875702619552612, 0.24217821657657623, -0.42306751012802124, -1.0957754850387573, 0.8461536765098572, -0.5255113244056702, -2.367011070251465, 1.4327218532562256, -1.8359808921813965, -1.0931503772735596, -0.9194725155830383, -0.11799110472202301, -0.17248764634132385, -1.0681891441345215, 0.4525175392627716, -0.7104922533035278, -0.05968938767910004, 0.05623801425099373, 1.8678147792816162, -0.9718713164329529, 0.15190783143043518, 0.31585773825645447, 0.036499328911304474, 0.5198699831962585, 1.9942103624343872, 1.7911981344223022, -1.160394549369812, 1.0838085412979126, 0.2344822883605957, 1.2037824392318726, 0.815737783908844, -0.9116550087928772, 0.6147889494895935, 0.8479849696159363, -0.13056136667728424, 0.7021542191505432, 0.14810654520988464, 0.07551731914281845, 0.08099603652954102, -1.8193496465682983, -0.8384332060813904, 1.3078726530075073, 1.3964035511016846, 1.0945994853973389, -0.7696781158447266, 1.1600061655044556, -0.2530868947505951, -1.4098907709121704, 0.643509030342102, -0.23505853116512299, -0.3499116897583008, 2.3901195526123047, -0.09281052649021149, 1.731758713722229, 1.1442326307296753, -0.19636212289333344, -1.211082935333252, 0.40944668650627136, 0.18514859676361084, 1.3799391984939575, -0.2590220868587494, -1.8939932584762573, 1.3550987243652344, -0.7750223875045776, -0.3961263597011566, 0.2742088735103607, 0.16299718618392944, 0.5239028930664062, 1.0357205867767334, 0.018861282616853714, 0.26473742723464966, -0.2445628046989441], [0.23178131878376007, 1.4172438383102417, -1.3652119636535645, 1.6097594499588013, -0.8991831541061401, 0.589180052280426, -1.2188901901245117, 0.2175845354795456, 0.09231062233448029, 0.8442816734313965, 0.26769545674324036, 0.6211209893226624, 0.6735236048698425, 1.17862069606781, 0.37502214312553406, 0.1214742586016655, -1.5910755395889282, -0.7249855399131775, -1.004166841506958, 1.1550049781799316, -0.07303670793771744, 0.0637957975268364, -0.5978967547416687, 0.410955548286438, -0.5127432942390442, -0.19706787168979645, -0.6698445677757263, 1.202826738357544, 1.4774810075759888, -0.1695719212293625, -0.48182153701782227, -0.07801160216331482, 1.7044891119003296, -0.345490962266922, -1.1024998426437378, -1.5001499652862549, -0.9265017509460449, 0.9789031147956848, -0.05188388377428055, 0.6339195966720581, -1.171373963356018, -0.3493568003177643, 1.1832817792892456, 0.04746723920106888, -0.765212893486023, -0.2510989010334015, -0.48520901799201965, 2.469106674194336, 1.3765696287155151, 0.9682562351226807, -0.7809298038482666, -0.9034296274185181, -1.760190725326538, 2.1051316261291504, -0.9462968111038208, 0.9682997465133667, 1.176032543182373, -1.019661784172058, -0.551400899887085, -1.4495271444320679, 1.4296045303344727, -0.1982996165752411, 1.4630732536315918, 1.0085316896438599, 0.0999201312661171, -0.863122284412384, -0.19867432117462158, -1.1307899951934814, -0.13497647643089294, -0.9746848940849304, -0.12635958194732666, -2.2662086486816406, 0.2826254367828369, -0.3816330134868622, -0.4319605231285095, -0.05797051265835762, 0.8417676091194153, -0.07446597516536713, -0.8210591077804565, -0.06059201806783676, 0.5645781755447388, 1.4979441165924072, 1.6481578350067139, -1.1539267301559448, -1.528472900390625, 2.5431790351867676, -0.3968636691570282, 1.0045830011367798, -0.7083439230918884, -0.27215704321861267, -0.6691429018974304, 0.003515851218253374, -0.7652174234390259, 0.4449104368686676, 1.124375820159912, -3.0016074180603027, -0.8093786239624023, -1.1365833282470703, -1.6264668703079224, 0.593245267868042, -0.25151747465133667, -0.2974971830844879, -0.3881479501724243, 1.203920841217041, 1.9791361093521118, -1.6854605674743652, 1.375561237335205, -0.07619608938694, -0.6703265309333801, -0.7713680863380432, -0.8404521346092224, 1.2443464994430542, 2.1596028804779053, 1.5737295150756836, -0.955053985118866, 0.12299209088087082, 0.9396288394927979, -0.06192455813288689, 0.7275600433349609, 0.3103369474411011, -0.3154570758342743, -0.12408752739429474, -0.5269889235496521, 0.3748806118965149, -0.7831026911735535, 0.8614454865455627, -1.761012077331543, 1.332898736000061, -0.6361772418022156, 1.0570470094680786, -0.18906977772712708, -0.841339111328125, 1.4008511304855347, 0.35518011450767517, 2.3289551734924316, -1.447479486465454, -1.8022034168243408, -1.4691542387008667, 0.38410478830337524, -0.0010952880838885903, 0.47428303956985474, -0.6628609895706177, -0.15721622109413147, 0.4036172330379486, -1.1440677642822266, -1.1574620008468628, 1.9103877544403076, -0.8101451992988586, 1.461388111114502, 1.3747481107711792, -1.3539247512817383, -1.5593047142028809, -0.8706747889518738, 2.9039998054504395, 0.958111047744751, 0.4906657040119171, -1.1996747255325317, -0.3188433349132538, 1.1057286262512207, -0.6845170259475708, -0.45311984419822693, 1.014526605606079, -1.2728893756866455, -1.170888900756836, -0.7334075570106506, 0.6600827574729919, 0.06981964409351349, 0.5351191163063049, -1.1358683109283447, -2.3082194328308105, 0.23581498861312866, -1.4897091388702393, -1.2202104330062866, -0.2810404598712921, -0.10398159176111221, -1.254257321357727, 1.330758810043335, 0.667561948299408, -0.006467539817094803, 0.22580847144126892, 1.033491611480713, -1.5043059587478638, -1.3137599229812622, 1.63506281375885, -1.2233017683029175, 1.112571120262146, 0.6899167895317078, -0.05993521586060524, -1.617533802986145, -1.5562851428985596, 0.09173063188791275, -0.9226462244987488, -1.681549072265625, 1.1801064014434814, 0.7840737700462341, -2.4110991954803467, 0.8573556542396545, -1.9005067348480225, -1.176680326461792, 1.298693060874939, 0.3960607647895813, 0.04478665068745613, 1.0162280797958374, 0.22569537162780762, -0.24754424393177032, -0.2825849950313568, 1.1783603429794312, 1.7091394662857056, -0.10830844193696976, 1.1919639110565186, 0.11929470300674438, -0.03365422785282135, 0.4895453155040741, -1.1455612182617188, 2.0153441429138184, 0.0169354397803545, -0.1757841259241104, 1.2415978908538818, 1.4362314939498901, 1.0390219688415527, -0.262020468711853, 0.592832624912262, 0.45970386266708374, 0.6821677684783936, -0.6574898958206177, 0.7179510593414307, 0.0697249099612236, 0.7907389998435974, -0.4201079308986664, -0.20561543107032776, 1.2197761535644531, 1.734326720237732, -1.2666170597076416, 0.620406985282898, -1.4742259979248047, 0.23218409717082977, 2.0848770141601562, 1.0137592554092407, 0.5115717649459839, -1.7584831714630127, 0.4665641486644745, -1.1485260725021362, 2.093467950820923, -2.398742437362671, 0.9266207814216614, -0.17940960824489594, 0.23590148985385895, -2.3472728729248047, 0.5944559574127197, -0.5835141539573669, -1.6794910430908203, -0.27484992146492004, 1.641793131828308, 1.1240628957748413, 0.4998929500579834, 0.90826416015625, -0.9723928570747375, 1.6673616170883179, 0.6890199780464172, 0.294560045003891, -0.015251540578901768, -0.3824807405471802, -1.4282370805740356, -1.0595084428787231, 0.6578218340873718, 0.16469931602478027, -0.011742400005459785, -0.39345473051071167, 0.4470059871673584, -0.1287878453731537, -0.46364301443099976, -1.6211557388305664, 0.1092996746301651, 0.6872036457061768, -0.5798235535621643, 0.19102288782596588, -1.3732644319534302, 0.2540995478630066, -0.9662511944770813, -0.4705313444137573, 2.2675249576568604, 1.3974543809890747, 0.4079172611236572, -1.716752290725708, -1.8509082794189453, 0.33714205026626587, -0.07088634371757507, 0.03719647601246834, -1.196445345878601, -0.9051764011383057, -1.073341727256775, 0.9205331206321716, -1.547432780265808, -0.3734265863895416, -0.3243328928947449, 0.7118163704872131, 0.01702234148979187, 0.7414902448654175, -0.7130894064903259, 2.442734718322754, 1.5778164863586426, -0.630363404750824, 0.7178478240966797, -1.8929357528686523, -0.5883796215057373, -1.0447391271591187, -1.0343111753463745, 0.15763702988624573, 0.10778505355119705, -0.2528306543827057, 1.2488080263137817, -0.5017057061195374, 0.7056556344032288, 0.08031310886144638, 2.1127355098724365, 0.1796397566795349, 0.3315940499305725, 0.8826678991317749, 0.5177404880523682, 0.5251930952072144, 0.8497717380523682, 0.08729367703199387, -1.8671166896820068, -0.6159763336181641, -1.0768296718597412, -0.13387669622898102, 0.8566381335258484, 0.02502303570508957, 0.9591835141181946, 0.756737232208252, -0.5070270895957947, 0.7332943081855774, 1.8010393381118774, -0.669100284576416, 1.111701488494873, -0.49349358677864075, -1.467360496520996, 0.6741980314254761, 0.8919445872306824, 2.673511266708374, -0.10323134064674377, -1.0800563097000122, 0.22089165449142456, -0.6705775856971741, -1.172088861465454, -1.0764210224151611, 0.26682353019714355, 0.3786860704421997, 0.2257813662290573, -1.228854775428772, -2.085969924926758, 0.9208396077156067, 0.4717480540275574, -0.28672727942466736, -1.813249945640564, -0.2603707015514374, -0.09286759048700333, -0.7685987949371338, -1.6308739185333252, -0.755052387714386, 0.6490601897239685, 1.4658435583114624, -0.20140515267848969, -1.8887603282928467, -0.1359534114599228, -0.9037583470344543, 0.5758929252624512, -0.3089545965194702, -0.37512850761413574, -0.06490921229124069, -1.538683533668518, 1.8992611169815063, -0.06057721748948097, 0.5481113791465759, -1.740095853805542, -1.6733367443084717, 1.0660271644592285, -1.3006384372711182, 0.789359450340271, 1.734326958656311, 0.09148648381233215, -1.519577980041504, -0.079127237200737, -0.8583247661590576, 1.4522924423217773, -0.9608781933784485, -0.8854042291641235, -0.4250948131084442, 1.9555182456970215, 0.1038801297545433, 1.1467546224594116, -0.050869666039943695, 1.7264758348464966, 0.9428262114524841, -1.2448867559432983, 1.2672107219696045, -0.5590153932571411, -0.2123098224401474, 0.03075658157467842, 0.025340506806969643, 1.114474892616272, 0.6287758350372314, 0.19814763963222504, -0.09793606400489807, 2.1110329627990723, -2.154491662979126, 0.7092825770378113, 0.2778564691543579, -0.03055698797106743, 0.3744640350341797, -1.3058825731277466, -0.13966727256774902, -0.6131219267845154, 0.7128488421440125, 1.2318148612976074, -0.600084125995636, 1.026831865310669, 0.4781922399997711, -0.07061900198459625, -1.5244672298431396, -0.6954962611198425, 0.38872912526130676, 0.1590714007616043, -2.12337589263916, 1.1460200548171997, 0.3119576871395111, 0.19607475399971008, 0.5744983553886414, 1.1257718801498413, 1.0586879253387451, -0.22988703846931458, 1.5013377666473389, 1.002192497253418, 1.297694206237793, -0.9965687394142151, 0.34290704131126404, 0.5911125540733337, -0.2251238375902176, 1.8102693557739258, 1.2107503414154053, -0.690917432308197, 0.6281905174255371, -1.6100797653198242, -0.301948219537735, -1.4782938957214355, -0.5654124617576599, 0.0299459770321846, -0.5367873907089233, -0.12199680507183075, -0.6281360983848572, 0.2722775936126709, 0.812612771987915, 0.09266311675310135, -0.30941876769065857, 2.214355945587158, 0.6099876165390015, 0.29327958822250366, -0.7579159140586853, -0.5412532687187195, 0.8577567338943481, 0.0555570013821125, -0.2877977788448334, 0.18791386485099792, -0.15162386000156403, -1.7112295627593994, -0.28480401635169983, -0.420749694108963, -0.03289605677127838, -0.17671716213226318, -0.28078004717826843, -0.5081998109817505, 0.4239823520183563, 0.016124505549669266, 0.04110195115208626, -0.9128485918045044, 1.06930673122406, 0.09723853319883347, 0.5478448867797852, -0.8804197907447815, 0.320696622133255, 2.3734891414642334, 1.6660178899765015, -1.1520706415176392, -0.021419787779450417, -1.807237148284912, -0.22686941921710968, 0.9095485210418701, 0.3069426417350769, 0.1564265787601471, 1.4110571146011353, -0.3278196156024933, 0.7661148905754089, -1.278706669807434, 0.3385713994503021, -0.2697557806968689, 0.4584944546222687, -0.3013755977153778, -1.3630988597869873, -0.052649594843387604, -0.14663538336753845, 1.3542742729187012, 0.9420769214630127, 0.6022921204566956, 0.21588416397571564, 0.04831229895353317, -1.0674093961715698, -1.8540717363357544, 0.8020052909851074, 0.438532292842865, 1.9774898290634155, 0.2675793170928955, -1.3804593086242676, -0.3824751675128937, -0.5071060657501221, -0.30686697363853455, -0.8736940622329712, -0.5549418330192566, 1.1328363418579102, 0.5781702995300293, 0.6137751340866089, -0.07397918403148651, 1.0122605562210083, 0.20386679470539093, 0.7277077436447144, -1.6921659708023071, -0.6609208583831787, -0.531734824180603, 0.3656854033470154, 0.5334211587905884, -1.2099449634552002, 0.7648834586143494, 3.281109094619751, 0.8895798921585083, 0.12997981905937195, 0.6596239805221558, -0.6686881184577942, -0.296733558177948, 0.25909578800201416, -0.38398557901382446, -0.7694747447967529, 1.2085075378417969, 0.5250391364097595, 0.21639330685138702, 0.5322604775428772, -0.4380951225757599, -0.8192169666290283, -0.5271693468093872, -1.0916637182235718, 1.2641589641571045, -0.4569093585014343, -1.3718231916427612, 0.6360125541687012, -0.3261280655860901, -2.040827989578247, -1.8788844347000122, -0.6926478147506714, 0.24588806927204132, -1.7064896821975708, 0.37652507424354553, 0.2160845249891281, -1.209537148475647, 1.4460952281951904, 0.6560662984848022, -0.42011258006095886, -0.4998067617416382, 1.289757490158081, 0.38149186968803406, 0.3557259738445282, -0.7687980532646179, 0.4805024564266205, -0.06827362626791, -0.04708223417401314, -0.33255502581596375, 0.559298574924469, -0.14698728919029236, -0.7105516791343689, -0.1761816442012787, 0.009801045060157776, 0.001844263868406415, 0.43266594409942627, 1.0053610801696777, 0.6212385892868042, 0.5760766863822937, -0.6509606838226318, -0.44865140318870544, 0.268279105424881, 0.24368281662464142, -1.2494089603424072, 0.2722587585449219, -0.482445627450943, -0.691490650177002, -1.2353594303131104, 0.14615486562252045, -0.5155434608459473, 0.12124723941087723, -0.21399372816085815, 0.9087140560150146, 0.897696852684021, 0.9471191763877869, 0.2506389319896698, 0.006019557360559702, -0.17602112889289856, -2.157045364379883, 0.8020769953727722, 1.2200307846069336, -0.41257187724113464, -1.1981531381607056, 1.1257810592651367, 0.2354455292224884, -0.5258383750915527, 0.7886773943901062, -0.4761190712451935, 0.44189363718032837, -0.37710675597190857, 0.8505448698997498, -0.34734195470809937, -0.17261594533920288, 0.31460484862327576, -1.4011423587799072, -1.6829735040664673, 1.9200032949447632, 0.6249760389328003, 0.417298823595047, -0.11837794631719589, -0.34547170996665955, 0.665273129940033, -0.27594900131225586, -1.7341148853302002, -1.1666324138641357, 0.25647634267807007, -1.1785337924957275, -0.34303149580955505, -1.1744006872177124, 0.2214413732290268, 1.4154362678527832, -1.0882768630981445, -0.3479701280593872, -0.39432820677757263, -0.03635535389184952, -0.453293114900589, -0.9235196113586426, 0.9222474694252014, 1.009265661239624, -0.6574935913085938, -0.5470944046974182, 0.18231156468391418, -0.1877957433462143, -2.199721097946167, 0.9632648229598999, 0.9551909565925598, 0.7066179513931274, -0.4372575283050537, -1.1756761074066162, 1.53072190284729, 0.7460185289382935, -0.39067983627319336, -0.6914443969726562, -0.5234003663063049, -0.9263377785682678, 1.3157758712768555, 1.2032933235168457, -0.6842573285102844, 0.14135904610157013, 0.7876154184341431, -0.6607149243354797, -0.25585296750068665, -0.44039323925971985, 0.3543335795402527, -0.5914239287376404, -1.756875991821289, -1.0038437843322754, -0.9474757313728333, 0.2513655722141266, 0.6513732075691223, 1.0907665491104126, 0.7529201507568359, -0.8674270510673523, 0.6538867354393005, -2.972698926925659, 2.206998109817505, -1.2331020832061768, 1.479472279548645, -1.3922768831253052, -0.33389368653297424, 0.1831531971693039, 0.5493234395980835, -0.32172831892967224, -0.933900773525238, 1.665510654449463, -0.393777459859848, 1.3807355165481567, -2.3456273078918457, 0.4211239218711853, 0.04137803614139557, 0.9219187498092651, -0.07853298634290695, -1.208651065826416, 0.37020471692085266, -0.15992406010627747, 0.3933919668197632, 0.9880037307739258, -1.075021743774414, -1.3361729383468628, -0.3055063486099243, 0.45033425092697144, -0.14329351484775543, 0.04946676641702652, -0.08353094756603241, -0.27184411883354187, -0.39320337772369385, 1.0241142511367798, 0.6474900245666504, 1.643215298652649, -0.7863027453422546, -1.4193542003631592, -0.02121259644627571, -2.4963603019714355, 0.6646777987480164, -1.1675275564193726, -1.7445579767227173, 0.27827706933021545, -0.14935995638370514, 0.5011367797851562, -0.5484501719474792, -2.3406124114990234, -2.057431697845459, -0.9777560830116272, -2.1179442405700684, -0.7200544476509094, -1.5705697536468506, -0.7457636594772339, -2.9624733924865723, -0.22232820093631744, 1.9646795988082886, 1.5205368995666504, -0.9396970272064209, -1.3131259679794312, -0.25195613503456116, -0.10077187418937683, 0.22485750913619995, -0.6183323264122009, 0.6009480953216553, 1.111869215965271, -1.1430798768997192, -1.6446852684020996, 1.034998893737793, -0.18507499992847443, -0.05768705531954765, 0.662053644657135, -0.7524334192276001, -1.156407117843628, 0.7529120445251465, 0.0586429238319397, -1.367143154144287, 0.22461901605129242, -0.16202546656131744, -0.8871462345123291, 0.6952378153800964, 0.27986034750938416, 0.1299782544374466, 2.023505210876465, 2.4591753482818604, 0.38961172103881836, 0.9229680895805359, 0.6248762011528015, -0.31644225120544434, -1.174699068069458, 1.1063534021377563, 0.6422466039657593, -0.7891650795936584, -0.004448120016604662, 0.9341230392456055, -0.5101369619369507, 0.5659291744232178, -0.16091710329055786, 0.04174364358186722, -0.34346890449523926, -1.5844365358352661, -0.9064416289329529, -0.19304259121418, -2.6824042797088623, 1.7461612224578857, 1.5900839567184448, -1.7195203304290771, -0.23940740525722504, -0.6080754399299622, 1.0093075037002563, -0.26146215200424194, -0.6371849775314331, 0.03020927868783474, 0.4563180208206177, -0.37314483523368835, 0.7809301018714905, -0.11158140748739243, -1.1643593311309814, 0.4116141200065613, 0.049072422087192535, -2.81387996673584, -0.22576433420181274, -1.8988680839538574, -2.1185975074768066, -1.103153109550476, 0.14236992597579956, 1.5389466285705566, -0.44386160373687744, -0.006737504620105028, 0.5876958966255188, 0.2723220884799957, -1.7758538722991943, 0.8294219970703125, 0.76401287317276, -0.4022178649902344, 2.08874773979187, 0.9157084822654724, -0.990668773651123, -0.06018276512622833, 1.5731064081192017, -1.5620455741882324, 0.34682586789131165, 0.49457061290740967, -0.1860838681459427, 1.9302204847335815, -2.5830624103546143, 1.9055523872375488, -0.7531930804252625, -1.2618889808654785, -0.8818120360374451, 0.0036723993252962828, 0.34988120198249817, -0.497175395488739, 0.2445836067199707, -2.2320451736450195, -1.431027889251709, -0.4719720780849457, -0.38833293318748474, -0.3551740348339081, 1.6882615089416504, 0.4580893814563751, -1.1050317287445068, -0.6613324284553528, -0.29634225368499756, -0.2887703478336334, 1.639991283416748, 2.2100930213928223, 0.22754305601119995, 0.3250870108604431, 1.1577106714248657, -0.09215645492076874, -1.5799530744552612, -0.16573074460029602, 0.20603814721107483, 0.48182493448257446, -0.8902401924133301, 2.900690793991089, 1.2695289850234985, -0.8447469472885132, 1.4377695322036743, 0.05521291121840477, -0.07528097182512283, -0.16435939073562622, -0.4127340316772461, -2.095285654067993, 1.142501711845398, -0.7879802584648132, -0.4758664667606354, -0.3477308750152588, 0.30938732624053955, 0.9785237908363342, 1.7154192924499512, -0.3381492793560028, -0.8241400718688965, -0.7795369029045105, -0.0862504094839096, 0.47303977608680725, 1.019260287284851, -1.1627411842346191, 1.3756065368652344, -0.23921850323677063, 0.6365242600440979, 0.8055113554000854, 0.08404731005430222, -0.871192991733551, -0.1076420322060585, 1.1238765716552734, -1.4675356149673462, -1.118536353111267, 0.10783255845308304, 2.1304996013641357, 0.7098375558853149, -0.00815738644450903, -0.008888994343578815, -0.6432082056999207, 0.6266089677810669, 0.722206175327301, 2.3121635913848877, -1.0759623050689697, -0.03575373440980911, 0.635733962059021, 0.31485021114349365, -1.8212672472000122, 1.0154019594192505, 0.4404376447200775, -0.26956284046173096, -3.118434429168701, -0.6958366632461548, 1.3559668064117432, 0.2523711025714874, -1.3211426734924316, -0.1906382441520691, -0.6411291360855103, -0.19161729514598846, -1.2350666522979736, -1.3415695428848267, 0.7566638588905334, -0.6108968257904053, -1.2656627893447876, 0.8724824786186218, -0.21964357793331146, 0.6817044019699097, -1.5542582273483276, -0.1780955195426941, 0.3587023615837097, -0.8894011974334717, 0.7229880094528198, 0.49467775225639343, -0.20523704588413239, 0.5370199084281921, -1.072226643562317, 0.21732757985591888, 0.04226618632674217, -0.7308292388916016, 1.2443604469299316, 0.0009373328066430986, 0.9665836691856384, 0.4830934703350067, 0.19655969738960266, 0.3048260807991028, -0.8614515662193298, 0.05873744934797287, -1.168343424797058, -0.2040131837129593, 0.3837093114852905, 2.075852394104004, -0.32724130153656006, -1.0925387144088745, 0.26519322395324707, -0.4803674817085266, 0.54021155834198, 0.34916070103645325, 0.5626486539840698, -0.10966183990240097, -1.0009963512420654, 0.3829117715358734, 0.4265362322330475, 0.7376421093940735, -2.713550329208374, -0.45132777094841003, -0.5852781534194946, -0.08103971183300018, -1.563247799873352, -0.3233708441257477, 0.051053766161203384, 1.8055858612060547, 0.5628677606582642, 0.34592917561531067, -0.03799280524253845, 0.6812791228294373, -0.3697259724140167, 0.833091139793396, -1.4015672206878662, 0.41984105110168457, -0.7501929402351379, -1.0610215663909912, 1.0760427713394165, -1.4827704429626465, 0.9385996460914612, -1.6156874895095825, 1.039384365081787, -1.0537030696868896, 0.25136643648147583, -1.2457756996154785, -0.4687179923057556, -0.11354868859052658, 0.5367016792297363, 2.1163265705108643, -0.14219693839550018, 1.0383176803588867, 0.3239659368991852, -2.0550456047058105, 0.8405465483665466, -0.22955021262168884, -0.19890344142913818, 0.14886055886745453, -1.6405192613601685, 0.008587859570980072, -1.557188630104065, -0.2651214599609375, 0.430496484041214, 0.183472141623497, -0.1703701764345169, -1.8093562126159668, -0.2834562063217163, -0.5173172950744629, -0.6008844971656799, 0.49170565605163574, -0.18913647532463074, 1.7724888324737549, 1.6607203483581543, -0.3848985731601715, 1.2602020502090454, -1.0464664697647095, 1.4414277076721191, 1.2431654930114746, -1.0357195138931274, -0.047828882932662964, 0.03502741456031799, 0.3134446442127228, 1.5243592262268066, -0.3692038357257843, 0.10522554814815521, 1.7147390842437744, 1.414289951324463, 1.4418468475341797, -0.3416896164417267], [-0.43060067296028137, 1.316672921180725, 1.728838324546814, 1.1580699682235718, -0.5213152170181274, -0.12533585727214813, 0.5402599573135376, -0.4519340395927429, -0.953011155128479, -1.359694480895996, -0.03742383047938347, -0.027070438489317894, -0.9225035309791565, -0.49370795488357544, -1.3983060121536255, 1.5816833972930908, -0.0522373802959919, 0.16055075824260712, 0.20913635194301605, 0.8526108860969543, -0.4726717472076416, 0.3694906234741211, 1.881790280342102, -0.3280024528503418, 0.9169513583183289, 0.25532564520835876, -1.1092976331710815, 0.9265774488449097, -1.1462260484695435, -0.5084531903266907, -0.49720290303230286, -1.0440778732299805, 0.6111562848091125, 1.3263508081436157, -0.96484375, 0.687721312046051, -0.3204953670501709, 1.626896858215332, 0.09211108833551407, 1.2811846733093262, -0.13147418200969696, -1.0023648738861084, 0.4818234443664551, -0.9639766812324524, -1.4918808937072754, -0.23976069688796997, 1.3703951835632324, -1.39042067527771, -0.714479923248291, -1.672812581062317, -0.1304369568824768, 1.4805748462677002, -0.2801936864852905, -0.631731390953064, -0.39229142665863037, -0.25422531366348267, -0.03998565301299095, 0.5147644281387329, 0.7600187659263611, 0.398033082485199, -2.7174112796783447, 1.0053681135177612, -2.2907145023345947, 0.6423170566558838, 0.5671825408935547, 1.346415638923645, 0.4634016156196594, -0.051731258630752563, 0.08583898842334747, 0.13787928223609924, -0.5807145833969116, 0.9799886345863342, -2.4881622791290283, 1.0897873640060425, -0.7716856598854065, 0.4454897344112396, -1.3923064470291138, -1.6795899868011475, -1.2178528308868408, -0.4167642295360565, -0.08323109894990921, 0.4882541298866272, -0.992355227470398, -1.2516579627990723, -0.046528469771146774, -0.4930786192417145, -0.12155496329069138, 0.8417035341262817, 0.700372576713562, -1.3859540224075317, 1.8858731985092163, 0.6142045855522156, -0.3994767367839813, 0.3571854531764984, -0.13735276460647583, 0.5482593178749084, 0.4818393886089325, -0.32987263798713684, 1.1503525972366333, -0.21066217124462128, 0.7457921504974365, -0.9258227348327637, -1.1149752140045166, 0.00986954290419817, 0.18474307656288147, -0.34509941935539246, 1.782749891281128, -0.07403028011322021, 0.6995166540145874, 0.15016056597232819, 1.05954909324646, 0.11119100451469421, -0.2553494870662689, 0.41853588819503784, -0.01000859122723341, 0.5161471366882324, 0.5986716151237488, -0.36470896005630493, -0.5877952575683594, -0.5121443867683411, 1.1335594654083252, 0.41436657309532166, -1.2450721263885498, -1.282509684562683, -0.7185679078102112, 0.7863042950630188, 0.40016624331474304, 0.7517694234848022, 0.2567574083805084, 0.10552238672971725, 0.5474616289138794, -0.5163009762763977, 0.06394484639167786, -0.04138753190636635, -2.0691938400268555, -1.1306132078170776, 0.2294638752937317, -0.9930461049079895, -2.1704607009887695, -0.6940444707870483, -0.18927650153636932, 0.7685927152633667, 0.48336610198020935, 1.2300126552581787, -0.2093135416507721, -0.9583331942558289, -1.2881711721420288, -0.824408233165741, 1.7892212867736816, 0.7875439524650574, 0.7685497999191284, 0.22104477882385254, -0.3122648000717163, -0.09918400645256042, -0.7460199594497681, 1.1816320419311523, 0.39472872018814087, -1.067915916442871, 1.1108906269073486, -0.643724262714386, -0.669563353061676, 0.27286067605018616, -0.554608941078186, 0.19927969574928284, -0.7050976753234863, 0.1586092859506607, -1.4096382856369019, -0.5673843622207642, 0.05940699949860573, -0.8523116707801819, 0.379037082195282, -0.03557982295751572, -0.33977010846138, 0.29449599981307983, -1.6105313301086426, 1.9863307476043701, -0.17123904824256897, -0.2084379941225052, 0.6847924590110779, -1.040008783340454, -0.9972862005233765, 0.06187783181667328, 0.3701426386833191, -2.046945571899414, 0.4282326400279999, 0.8155454397201538, 1.4306102991104126, -0.8869991302490234, -1.0153262615203857, 0.7591317296028137, 1.111801266670227, 1.057959794998169, 0.635129451751709, -0.1611032783985138, 0.3914933502674103, 0.5295983552932739, 1.8100122213363647, 0.9764610528945923, 0.04316118732094765, -2.788299798965454, -1.8669356107711792, 1.7879128456115723, -0.677873432636261, 0.372457891702652, -2.0824697017669678, 1.9752639532089233, -0.20437780022621155, -2.061577081680298, 0.05865517258644104, 0.4872361421585083, 0.021730517968535423, 0.2542341351509094, 1.8148155212402344, 1.3716259002685547, -0.5126463174819946, 1.6042778491973877, 1.2658237218856812, -0.5717016458511353, -2.138035774230957, 2.2065980434417725, 0.916253387928009, -2.1992549896240234, -0.9966614246368408, 2.0108742713928223, 1.5190750360488892, -0.62510746717453, 0.9774395227432251, -1.7907735109329224, -0.14815574884414673, 0.6834273338317871, -1.0377088785171509, 1.7051277160644531, 0.05252932757139206, -0.5906276702880859, 0.14030537009239197, -0.9231277108192444, 1.238672137260437, 1.5472146272659302, 0.7774605751037598, -0.517634928226471, 0.005205250345170498, -1.2655409574508667, -1.6638532876968384, -0.02001422457396984, -1.3842960596084595, -0.1504260152578354, 0.9940505027770996, -0.02040915936231613, 0.13752305507659912, -0.7624345421791077, -0.22508162260055542, 1.2581188678741455, -0.6044061779975891, -0.280261367559433, 0.008841299451887608, 0.8835163116455078, 0.42380791902542114, -0.6279005408287048, -0.8479564785957336, 0.9598945379257202, 1.3602112531661987, -1.5606461763381958, -0.022937947884202003, 1.206784725189209, 1.1148844957351685, 0.6820855140686035, -1.0707738399505615, -0.45594045519828796, -1.4776544570922852, -0.42995068430900574, 0.4149633049964905, -0.6748865842819214, -0.2700497508049011, -1.5677595138549805, 1.4065953493118286, 0.47467368841171265, -0.12096986174583435, 1.1424479484558105, 0.805861234664917, 2.329180955886841, -1.8385649919509888, -0.30780282616615295, 0.9836966395378113, -0.6028185486793518, -2.0503592491149902, -0.22825221717357635, 0.8384150266647339, -1.123596429824829, -0.9392350912094116, 0.5297387838363647, 0.543140709400177, -1.744834542274475, -0.2727615535259247, -0.5234119892120361, -0.7817682027816772, 1.1383328437805176, -1.5407625436782837, -0.030475661158561707, 0.13941332697868347, -0.9941057562828064, -0.1811821162700653, 1.8046585321426392, 0.5102740526199341, -0.11876598000526428, -1.5842241048812866, -0.41590309143066406, 1.1216182708740234, 1.519201636314392, 0.4914354383945465, 0.19270290434360504, -0.6331637501716614, 0.045038238167762756, 0.8795175552368164, 0.3776366114616394, -0.33067983388900757, -0.7826336026191711, -0.6684545874595642, -1.1400244235992432, 1.452893853187561, 0.1612989455461502, 0.11617254465818405, -0.2761174142360687, 0.49845898151397705, 0.0951119139790535, -0.27380093932151794, 0.627903938293457, -1.3289210796356201, 0.544617235660553, 0.918319821357727, 1.3278690576553345, -0.1862107515335083, 0.07507573068141937, 1.3340696096420288, 0.3565676808357239, -0.22187162935733795, -1.2485324144363403, -0.5134897232055664, 0.9231265187263489, 0.23940639197826385, -0.4483034908771515, 0.30282026529312134, 0.9668344855308533, -0.4113171696662903, 1.3591687679290771, 0.9011030197143555, 0.19257549941539764, -1.2989511489868164, -1.4265358448028564, 0.4690861701965332, -0.07783681154251099, -0.5291764736175537, -1.1728569269180298, -0.6847977638244629, 0.2649833858013153, 1.181317687034607, -3.019084930419922, -0.9858250021934509, 0.6385895013809204, -1.7762163877487183, 1.1960817575454712, 2.0854220390319824, 0.11663389205932617, -0.1530393660068512, -0.38307154178619385, -0.5571640729904175, -0.1934729367494583, -1.1216224431991577, -1.7100435495376587, -0.649491548538208, -0.4363129138946533, 0.38210001587867737, 0.9671837091445923, 0.1277141273021698, -0.05793387442827225, 1.0309046506881714, 0.7634948492050171, 0.15415982902050018, 0.26151540875434875, 0.5621052384376526, -0.9937760829925537, -0.49514833092689514, -0.5020994544029236, -1.2250057458877563, 0.7457435131072998, -1.814699649810791, 1.0033172369003296, 0.3784123957157135, 0.9591859579086304, -0.5007048845291138, -0.20920604467391968, -1.31309974193573, 0.4143674969673157, -0.26875069737434387, 1.9593076705932617, -0.36542996764183044, 0.8511014580726624, -1.167019248008728, -0.1666433960199356, 1.5700469017028809, 0.17533382773399353, -1.4485526084899902, -0.12622246146202087, 0.3811286389827728, -0.64899080991745, 0.8861129879951477, 0.8369271755218506, 0.37761738896369934, 1.106974720954895, -0.7765004634857178, -1.4876418113708496, 0.21077120304107666, -0.4148610234260559, -0.6187649369239807, -0.1489109992980957, -0.8201048970222473, -1.1165369749069214, -0.01416381262242794, 0.5702818632125854, 0.9744170308113098, 0.2543388903141022, -0.8994331955909729, -1.1017118692398071, -0.5191602110862732, -0.2770524024963379, -0.9402945041656494, -0.20943112671375275, 0.915009617805481, -1.7421926259994507, -0.2017355114221573, 0.6542190313339233, 0.2799190282821655, -1.2240740060806274, 0.4826914370059967, 2.4468841552734375, 0.0490020327270031, 0.4077329933643341, 0.8218658566474915, -0.3487013578414917, 0.06283725798130035, 0.6214327216148376, -2.2736170291900635, -0.27252018451690674, -0.2655925154685974, 0.2063901275396347, 0.9841211438179016, 0.8240370750427246, 0.23682837188243866, -0.37296611070632935, 1.0711731910705566, -1.0702159404754639, -0.35936084389686584, -1.808234453201294, -3.0591306686401367, 0.7539252042770386, 0.03588435426354408, 0.9907863140106201, -1.1817353963851929, -0.35468798875808716, 0.8418912291526794, -1.118991732597351, -1.5601264238357544, 1.6937083005905151, 0.04228604584932327, 0.613572895526886, -0.5291654467582703, -0.5489627718925476, -2.4011127948760986, 0.5500668883323669, 0.22754928469657898, 1.3336237668991089, 0.38866060972213745, -0.8530365228652954, 0.021304473280906677, -0.5676158666610718, -0.26803338527679443, -0.293936550617218, -0.13726229965686798, 0.07726547867059708, 0.33982840180397034, -0.9417636394500732, 0.6223350167274475, 0.2500814199447632, 2.187128782272339, -2.038104772567749, 1.667585849761963, -1.3184350728988647, 0.6533240079879761, 0.4218047559261322, 0.4139798879623413, 0.14316371083259583, 0.6111899614334106, 0.9625533223152161, 0.05224457383155823, -0.5178559422492981, -0.6068138480186462, -0.5144819021224976, 0.9847967028617859, -1.0319842100143433, -2.5747015476226807, -2.7712666988372803, 0.2703697085380554, -0.426296591758728, -0.048315517604351044, -0.7509887218475342, -3.409632921218872, -1.3726595640182495, 0.16429020464420319, -0.042889248579740524, -0.02022724784910679, 0.07212483137845993, -1.8666579723358154, 1.4629552364349365, -1.080437421798706, 0.06520560383796692, 0.8221800923347473, -0.45370060205459595, -0.34832555055618286, -0.9628250598907471, 0.1088358461856842, -1.077640175819397, -0.1487882137298584, 0.6200504302978516, -0.30363282561302185, 0.7847791314125061, -1.3183059692382812, -2.3572516441345215, 0.27248093485832214, -0.41014567017555237, -0.9200296998023987, 1.0407832860946655, 1.3425477743148804, -0.18237289786338806, -0.3469749987125397, 1.2449837923049927, -0.7931627035140991, -0.20256391167640686, 0.7938202023506165, -0.18573516607284546, 0.7294670343399048, 1.442619800567627, -1.5478456020355225, 0.1674339473247528, -0.6599169373512268, 0.7463632822036743, -1.8996248245239258, -1.3027887344360352, -0.9360974431037903, 0.5910627841949463, 0.7572604417800903, 0.6910470724105835, 0.1686272770166397, 0.024736715480685234, 0.43444210290908813, -0.3501497209072113, 0.10431582480669022, 0.4977354109287262, 0.06789735704660416, -0.8674493432044983, 1.2654956579208374, -0.3616569936275482, -0.13184763491153717, -0.8300677537918091, -0.13500574231147766, 0.4012491703033447, 1.518206000328064, -0.8793883323669434, -0.8512457609176636, 0.6253819465637207, 0.40037602186203003, 0.061726171523332596, -0.16313035786151886, 0.6540594696998596, 1.2572492361068726, 0.019224736839532852, 0.4712236821651459, -0.38694584369659424, -1.0451122522354126, 2.376901388168335, 0.06831023842096329, -0.764351487159729, -2.8273632526397705, -0.7428452968597412, -2.0503900051116943, -0.34745556116104126, 1.0547459125518799, -0.15621474385261536, -0.9068684577941895, -1.3134260177612305, -1.8679804801940918, -1.1795017719268799, -1.3267642259597778, 0.6209706664085388, 0.6998708844184875, 0.677500307559967, -2.0062665939331055, -0.7652342915534973, 0.3605729639530182, 0.4740475118160248, 1.604124665260315, 1.884084701538086, -0.5946375131607056, 0.3321758508682251, 0.8871495723724365, 1.1324576139450073, 1.0437517166137695, -0.5164677500724792, 0.7055037021636963, 1.6001458168029785, -0.07592334598302841, -0.13640999794006348, 0.8921693563461304, 0.026378046721220016, -0.18629981577396393, -0.6949535012245178, 0.6240076422691345, -0.9037836194038391, -0.5095565319061279, 0.6059984564781189, 1.7527976036071777, -2.668952226638794, 0.943915843963623, -0.4358687102794647, 0.1700258105993271, 0.9555037617683411, -1.9043234586715698, -1.0364429950714111, -1.1176071166992188, 0.16485495865345, 0.5716584324836731, -0.49450960755348206, -0.4275290071964264, -0.9598792791366577, -1.4794907569885254, -1.079695701599121, -1.9688358306884766, -0.5505838394165039, -1.9938627481460571, -0.20685884356498718, -0.11599604040384293, 0.20148172974586487, -0.18606074154376984, -2.462711811065674, 2.8790597915649414, -1.2645491361618042, 1.0274159908294678, 0.6261233687400818, 2.033684015274048, -0.6980108022689819, 0.18903256952762604, 1.7106469869613647, 0.021904585883021355, 1.1415092945098877, 1.0150468349456787, 1.1759158372879028, -1.4142687320709229, 0.160095676779747, -0.28638362884521484, -0.6337279081344604, 0.998059093952179, -0.4223695993423462, -0.28197261691093445, -0.06311019510030746, 0.04394614323973656, -0.6847529411315918, -0.08198890089988708, -1.9595746994018555, 0.24756446480751038, -0.30716782808303833, -1.993451714515686, 0.30912140011787415, -0.38902777433395386, 1.094774603843689, 0.6823177337646484, 0.7282397150993347, 0.6831696033477783, 1.3147443532943726, -0.08176466822624207, 0.4243481755256653, -1.4280004501342773, 1.1628822088241577, -0.2819299101829529, 0.47298160195350647, -1.5878872871398926, 1.0402305126190186, -0.502319872379303, 1.108008861541748, -1.1390756368637085, -0.47877028584480286, -1.0767656564712524, -0.1517128050327301, -0.8500937223434448, 0.32606709003448486, 0.9755672812461853, -0.23164834082126617, -1.2648026943206787, 0.1938108205795288, -0.3686404526233673, 1.3199669122695923, -0.4948026239871979, -1.7440634965896606, -1.0968984365463257, 0.19160096347332, -0.32152998447418213, -0.7240505218505859, -2.839242458343506, -0.555160403251648, 1.9070137739181519, 0.26233190298080444, 1.6822086572647095, 0.14814291894435883, 0.41888880729675293, 0.4708387851715088, 0.6656498312950134, -0.973913848400116, 1.4060957431793213, 0.8302369117736816, 0.45945286750793457, 1.0899152755737305, 0.631713330745697, 2.028822183609009, -1.8773982524871826, 0.5866736173629761, -1.0585277080535889, -0.7806486487388611, -0.89447021484375, 0.35383594036102295, -0.49443069100379944, -0.07221437245607376, -1.0696851015090942, 0.6626917719841003, 0.01316814124584198, 1.4695225954055786, 1.2623279094696045, 1.3405187129974365, -0.39424213767051697, -0.256436288356781, 0.5840730667114258, 0.052131764590740204, -1.5738154649734497, 0.6905118823051453, -0.33624809980392456, 1.4976006746292114, 0.7746481895446777, 0.39705008268356323, 1.1567710638046265, -1.4516234397888184, 0.288003534078598, 0.17899726331233978, -1.1461735963821411, -0.3142376244068146, -0.3853398263454437, 0.2861131429672241, -0.7855236530303955, -0.2593628764152527, -0.5622411966323853, 2.027911901473999, 0.14097444713115692, 0.579060435295105, -1.5117955207824707, -0.738331139087677, 0.11960496008396149, 0.8025268912315369, 0.47366198897361755, -0.27035388350486755, 0.2638445794582367, -1.0918145179748535, 0.2785218358039856, -0.12501269578933716, -2.117445707321167, -0.15756657719612122, -2.09340763092041, 0.7915856242179871, 0.45887887477874756, 0.3654303252696991, -0.6804999709129333, 1.8620831966400146, -0.3731813430786133, -1.0680979490280151, -0.5663148164749146, -0.45923954248428345, 0.9536531567573547, -0.7764609456062317, 0.02986905537545681, 0.8780462741851807, -1.1663720607757568, -0.9425331950187683, -0.16703830659389496, -0.00911185797303915, 0.8777336478233337, 0.5673866271972656, -0.7503606081008911, -0.07141359150409698, 0.9411683678627014, 1.751029133796692, -0.6275917887687683, 2.504855155944824, 1.5418004989624023, 0.5366455316543579, 0.013809267431497574, -0.3121059834957123, 0.7065423130989075, 0.41489264369010925, -1.3474252223968506, 0.8672551512718201, -0.5807145833969116, 0.050105899572372437, 0.7634435296058655, 1.4297676086425781, -0.9143177270889282, -1.1447534561157227, -0.03782450035214424, -0.4024953246116638, -0.05643995478749275, 1.2444792985916138, -0.10998276621103287, -0.7252506017684937, 0.922782301902771, 0.4594985246658325, 0.7747766971588135, -0.7162995338439941, 1.1567529439926147, 1.2664425373077393, -1.075403094291687, -1.8010162115097046, 0.020604094490408897, -1.0739415884017944, 0.412577360868454, -1.0025873184204102, 0.9701057076454163, -0.8710746765136719, -0.7963414192199707, 1.7526427507400513, -0.4039354622364044, 1.0561257600784302, -2.064115285873413, -0.26952555775642395, 0.5220786929130554, 0.2840677797794342, 0.10230492055416107, -0.021697349846363068, 0.04351598769426346, -0.05360463634133339, -0.5202382206916809, 0.3101881146430969, 1.2846510410308838, 0.9474731087684631, -0.2606929540634155, 0.1001155823469162, -0.7820956110954285, 0.5541068315505981, 0.6397593021392822, 0.29275768995285034, 0.6640205383300781, 1.4808377027511597, 1.866709589958191, -0.01502304244786501, 0.14532293379306793, 0.6828727722167969, -1.1377238035202026, -0.8024067878723145, -1.5311534404754639, 1.1068047285079956, -0.5025590658187866, 1.4885162115097046, 1.5839219093322754, 0.29845964908599854, 0.17358888685703278, -0.6876769661903381, 0.3751378357410431, 0.08784713596105576, -1.5023133754730225, -0.6309452056884766, 0.49563127756118774, 0.6939929127693176, -0.9837760925292969, 1.8860974311828613, 1.1519920825958252, -2.1050302982330322, 0.537895679473877, 0.06185706704854965, 0.2870899438858032, 0.22722159326076508, -1.0575320720672607, -1.8054710626602173, 1.1436734199523926, 0.45110249519348145, -0.19134531915187836, 0.005003224592655897, 0.9442871809005737, 0.4127441346645355, 0.41019824147224426, -0.9773122072219849, 0.48607686161994934, -0.19330625236034393, -0.27905556559562683, 1.4857001304626465, 1.6872624158859253, 0.5850944519042969, -0.32303518056869507, -1.2921391725540161, 1.520904541015625, 0.5389326214790344, 0.47879257798194885, -0.3346283733844757, 0.5183904767036438, 0.4112658202648163, -0.5324298739433289, -0.8083155751228333, -0.25381356477737427, -1.456610918045044, 1.6030282974243164, -0.4002284109592438, 0.20236696302890778, 0.3425420820713043, -0.633467972278595, 0.07429493963718414, -1.27695894241333, -0.9870663285255432, -0.20461219549179077, 1.216407060623169, -3.2257659435272217, -1.6564902067184448, -2.057887077331543, 1.3932571411132812, -0.2516454756259918, 1.20369291305542, 0.9360543489456177, -0.5501821637153625, 1.3937842845916748, -1.3733372688293457, 0.43011143803596497, 0.9089357852935791, 0.800586462020874, 0.8512473702430725, 0.8813149333000183, 2.0912318229675293, -0.6130764484405518, 0.34534454345703125, 1.274946689605713, 0.5119450688362122, 0.04922229051589966, -1.3130226135253906, -0.2919484078884125, 0.23547132313251495, -0.31493473052978516, 0.5354546904563904, -0.20592302083969116, -0.23575614392757416, 0.33267736434936523, -1.736323356628418, 0.01164037361741066, -0.2977975308895111, -3.407421112060547, 1.5973875522613525, -1.5347979068756104, -0.6736250519752502, 2.1966874599456787, 0.011763137765228748, 0.25237905979156494, 0.28168192505836487, 0.891342043876648, -0.5083163976669312, 1.454243540763855, -0.8270573616027832, 0.2665346562862396, -1.0675890445709229, 1.2012708187103271, 0.21665790677070618, -1.0597450733184814, -0.7605801224708557, -0.3806687295436859, -1.6418203115463257, -0.5976378321647644, 0.08074916154146194, -0.4609065055847168, 1.0457804203033447, 0.5655132532119751, -0.28892529010772705, -0.060798171907663345, -0.3184554874897003, -0.4640887975692749, 0.8039271235466003, -1.2356334924697876, -0.5560652017593384, 1.6397054195404053, -0.7741234302520752, 0.8200406432151794, 0.08307816088199615, -0.29249653220176697, 1.0367316007614136, -0.5567129254341125, 1.8029030561447144, -0.8794788122177124, 0.16547460854053497, -1.5734179019927979, -1.6005373001098633, 1.5025075674057007, -0.4040035009384155, 1.5001665353775024, 1.5333054065704346, -0.017661280930042267, -0.07944110035896301, -0.0686464011669159, -1.6381173133850098, -0.14840565621852875, 1.448606252670288, -0.25817936658859253, 1.1767349243164062, -0.8508396744728088, 0.6845322847366333, 0.17566071450710297, 0.04915355145931244, -0.8256951570510864, 0.14655716717243195, -1.2866427898406982, 0.14955243468284607, 0.5742796659469604, 0.5354570746421814, -0.6009677052497864, -0.579892635345459, 0.5676246285438538, -0.05990675464272499, -0.3739910423755646, 1.3070391416549683], [-0.5889392495155334, -1.357127070426941, -1.769675374031067, 0.04958725720643997, 0.1407862901687622, 0.04076162725687027, -0.6867699027061462, -0.5477208495140076, 0.11236990988254547, 0.8877072930335999, 1.6172384023666382, 0.3466307818889618, -0.058564770966768265, -1.3177590370178223, 0.1472872942686081, -0.31771039962768555, -0.798607349395752, -0.13905055820941925, 0.2950003743171692, 0.037503160536289215, 0.0071615115739405155, -0.8097851872444153, -1.2660380601882935, -1.7873218059539795, -0.4967341423034668, 0.16643041372299194, -0.05995062738656998, 0.3460519313812256, 0.1264175921678543, -0.48637279868125916, 0.5967673063278198, 0.8366944789886475, -0.17108505964279175, -0.39388176798820496, 1.4691545963287354, 1.8410392999649048, 0.01153502520173788, -0.7123574614524841, -0.42378297448158264, -0.45982691645622253, 0.023051738739013672, 0.09833931922912598, -0.13123007118701935, 0.3554011285305023, 0.9677364826202393, 0.8486162424087524, -0.12847217917442322, 1.1968828439712524, 1.3748583793640137, 0.27530694007873535, 0.4302782714366913, -0.34788283705711365, -0.08730503171682358, -0.015811314806342125, 1.1293015480041504, -0.8759319186210632, -2.451679229736328, 0.6034270524978638, -0.43341177701950073, -0.3631257712841034, -0.749919593334198, -0.8104599118232727, -0.7037079930305481, 0.9236518144607544, -0.8495209813117981, -0.6467918157577515, 0.22980919480323792, -0.219014972448349, -1.0449098348617554, 0.08934953063726425, 0.7469559907913208, -0.41312456130981445, 1.7143807411193848, -1.8073002099990845, 1.566726565361023, -0.31384024024009705, 2.339162826538086, -0.8358849883079529, -2.137718439102173, -0.27878522872924805, 1.0470482110977173, 0.3362441956996918, 0.28129130601882935, 0.09373693913221359, 0.3743189871311188, -1.137941598892212, 1.4375073909759521, -0.09686709195375443, -0.9164921045303345, -1.144814372062683, 1.4741812944412231, 1.7546794414520264, -2.08896541595459, 0.27603864669799805, -1.1316694021224976, -0.06394648551940918, 0.16242453455924988, -0.45771515369415283, 0.940071702003479, -1.1484220027923584, -0.6940826773643494, 0.19643352925777435, -0.8881314992904663, 0.4828861951828003, -1.4642038345336914, 0.9469189643859863, -1.5980411767959595, -0.6610959768295288, -0.27645865082740784, 1.1557796001434326, -1.241607904434204, 0.6781367659568787, 0.9403877258300781, -0.31971004605293274, 0.41883835196495056, -1.4925627708435059, 0.963054895401001, -0.47026440501213074, -1.0074777603149414, 2.2147216796875, -1.670627474784851, -1.1623005867004395, 1.0692334175109863, 0.08596497029066086, -1.3150395154953003, -1.1576625108718872, -0.938343346118927, -0.010591461323201656, 0.15465904772281647, -0.8155076503753662, 0.4941153824329376, 0.8142823576927185, -0.32568618655204773, 3.2632081508636475, -1.4049662351608276, -0.6536506414413452, -0.9872457981109619, 0.1847585290670395, 0.7174291014671326, 1.0128918886184692, -0.08272229880094528, -1.6916790008544922, 0.5849438905715942, 0.010675275698304176, 0.6393592357635498, 0.5117878913879395, -0.05689368024468422, -0.6328389048576355, -2.083209753036499, -0.4460381269454956, -0.432614803314209, 1.575032114982605, -0.11130112409591675, 1.4078234434127808, 0.4737250804901123, 0.6361314058303833, 2.0254933834075928, -1.128983736038208, 0.32535457611083984, -1.4717305898666382, 0.49029290676116943, 0.41525691747665405, 0.11233542114496231, 0.4065645933151245, 0.3974366784095764, 0.5857453346252441, -0.18538261950016022, 1.0789682865142822, -0.17944593727588654, 0.19931063055992126, 0.19800586998462677, 0.442786306142807, 0.0641297921538353, 0.40379074215888977, 0.1808348447084427, 1.235209584236145, -0.4172760844230652, -0.05969366058707237, 1.4724805355072021, -0.354037880897522, 0.4575386345386505, -0.4255169630050659, 1.674496054649353, -1.047094464302063, -0.20368798077106476, 0.5456641316413879, 0.39778339862823486, 0.7118152976036072, 1.00837242603302, 1.0291633605957031, 0.39336052536964417, 0.5307667851448059, -2.212256908416748, -1.154438853263855, 0.27883219718933105, -0.4334428906440735, 0.5536797642707825, -0.5419092774391174, -1.4020776748657227, -0.13294009864330292, 1.5579437017440796, 1.4494071006774902, 1.5086939334869385, 1.245327115058899, 0.31283172965049744, 1.8776898384094238, -0.8668836355209351, 0.951947808265686, 2.0969669818878174, 0.9508904814720154, -1.1767606735229492, -1.421313762664795, -0.5935678482055664, 0.5565592646598816, -1.4599144458770752, 0.62047278881073, -0.6695454120635986, -0.21246038377285004, -0.6593029499053955, -1.0825552940368652, 1.0526933670043945, 0.1926197111606598, 0.13077065348625183, -1.8266100883483887, -1.0973834991455078, 0.975192129611969, -1.5719196796417236, 1.1699062585830688, 1.0737687349319458, -0.6288946866989136, -0.9419109225273132, 1.2221869230270386, -0.9903250932693481, 0.22470109164714813, 1.208851933479309, 0.7124443054199219, -0.643632709980011, -0.0018505267798900604, -0.7347150444984436, 1.2939332723617554, 0.22058998048305511, -0.40678560733795166, 0.020883144810795784, -0.6789058446884155, 1.0045865774154663, -0.1339489370584488, 0.061623379588127136, -0.6459769606590271, 0.3717885911464691, -0.4840766489505768, 1.0882400274276733, -0.3645164966583252, 1.3585224151611328, -0.4748336970806122, 0.28315186500549316, -0.002628308953717351, 0.6489367485046387, -0.976840078830719, -0.013756802305579185, -2.169023036956787, 0.4232853949069977, -0.5236007571220398, -1.534652590751648, 0.6561451554298401, 0.11181098222732544, -1.3395637273788452, -0.8667557835578918, 0.09872529655694962, 0.1513521373271942, -1.3346184492111206, -1.2913060188293457, 0.9121254682540894, -0.7997691631317139, -0.18939711153507233, -0.4595373868942261, -0.9028570055961609, 0.2653566300868988, 0.22252774238586426, -0.680054783821106, -1.8944250345230103, 1.2107594013214111, -0.4163670539855957, -0.7432336807250977, 0.9479705095291138, -2.7834532260894775, -1.185645580291748, 1.5700874328613281, -0.8556539416313171, 0.44116541743278503, 0.9790951013565063, -0.2847258746623993, -0.047617003321647644, -0.6011924147605896, -0.4530707895755768, 0.28165626525878906, 0.2137645035982132, -0.25477665662765503, -1.1004952192306519, -0.5762641429901123, 0.8161184787750244, 1.4392801523208618, -0.8046440482139587, -0.623669445514679, 0.49191492795944214, 0.21722425520420074, -2.6781439781188965, -1.6017258167266846, 0.03775448352098465, -0.10556013882160187, -1.035405158996582, -1.4473546743392944, -1.4361350536346436, -1.6266707181930542, 1.1555681228637695, -0.05442700535058975, 0.7497053742408752, -1.1362329721450806, 0.033267222344875336, 0.7773261070251465, 0.44268280267715454, -0.4349600374698639, -0.5599722266197205, 1.6955888271331787, -0.44005244970321655, -1.5153263807296753, -0.27433016896247864, 1.6662542819976807, 0.36312127113342285, -1.6078860759735107, 0.029170697554945946, 0.10160592943429947, -0.7439552545547485, -1.1570677757263184, -0.6888831853866577, -1.4590829610824585, -0.857768714427948, -1.1087076663970947, -1.3818126916885376, 0.7191902995109558, -0.9560146331787109, -0.8634375929832458, -0.49288779497146606, 0.3394506871700287, 0.3910806477069855, -0.866393506526947, -0.39273351430892944, -0.44856637716293335, -1.1953996419906616, 0.5478547215461731, 0.4895167648792267, 0.8843532204627991, -1.182452917098999, -1.2786083221435547, 0.38641127943992615, -0.12649475038051605, -0.14319711923599243, 1.1567450761795044, 0.006172491703182459, -0.9044350385665894, -1.4629895687103271, -0.19662339985370636, 0.14870809018611908, 0.48343825340270996, -0.5044912695884705, -0.7013325691223145, -1.1183841228485107, 0.3479025661945343, -0.5029821991920471, -0.10528712719678879, 1.0631000995635986, 0.005851570051163435, -0.6862705945968628, -0.9934279322624207, -0.4699791371822357, 1.6869392395019531, -0.7349393367767334, 0.28943997621536255, 0.9933850169181824, 0.10686265677213669, 1.714840292930603, -1.337955117225647, 1.159916877746582, -0.5795800089836121, 1.743706226348877, -0.5974475741386414, 0.33612847328186035, 0.11553605645895004, -1.5212087631225586, -1.688879370689392, -0.9848881959915161, -1.1316325664520264, 0.565574586391449, -0.7528790831565857, -0.5225262641906738, 0.026934737339615822, 1.1953731775283813, 0.49720385670661926, 0.16424033045768738, 1.6067712306976318, -0.8896193504333496, 0.323805570602417, 0.23572443425655365, -0.44279786944389343, 0.8087710738182068, -0.23675209283828735, 0.27596065402030945, -0.4508378505706787, -1.8254703283309937, 0.4614836573600769, 0.7419121265411377, -0.26723968982696533, -0.6032019853591919, 0.5122140645980835, 1.2462234497070312, -0.5629517436027527, 1.309307336807251, -0.8862667083740234, -0.23448576033115387, 0.31924009323120117, -0.313067764043808, -0.00010485570965101942, -0.12396559119224548, -0.34862077236175537, 0.1750364899635315, 0.6261885166168213, -0.04103871062397957, -1.0856513977050781, 0.712519109249115, 1.6210846900939941, -0.8206517100334167, 1.3933453559875488, 0.3247631788253784, 0.7044650316238403, 0.6985384225845337, -0.0651869922876358, -0.2810412049293518, -0.7223846912384033, -0.9005900621414185, 1.08455491065979, -0.7403624653816223, 0.7519218325614929, 0.5455575585365295, -0.3042503893375397, -0.5489364862442017, -1.2362264394760132, -0.3456011116504669, -0.2729467451572418, 0.4375087320804596, -0.3449733257293701, -0.9514232277870178, 0.538055956363678, 2.1227211952209473, 1.3146116733551025, 0.3530888855457306, -1.8834275007247925, 0.3948625922203064, -2.217113971710205, 1.00441575050354, -0.5045733451843262, -0.2893955111503601, 0.7962194085121155, -1.897416591644287, -2.546660900115967, 0.019921958446502686, -0.038191623985767365, -1.2291045188903809, 1.3293081521987915, 0.06911632418632507, 0.14908380806446075, 0.22220496833324432, -0.1044926568865776, -0.21272172033786774, 1.366570234298706, -0.39421138167381287, 0.7574899792671204, 1.6227116584777832, 1.3166577816009521, 0.5848164558410645, 0.9677969813346863, -1.1215499639511108, -0.6403235793113708, 0.4535394012928009, -2.721818685531616, 0.6049599647521973, 0.36082255840301514, 1.082401156425476, -0.12694863975048065, -0.1591048538684845, -0.831280529499054, -1.1172281503677368, 1.4263626337051392, -1.0118916034698486, -0.25681155920028687, -0.7649151682853699, -0.2970700263977051, 0.8921700716018677, -0.03131768852472305, 0.45939329266548157, 0.667212963104248, -0.9544507265090942, -0.6352831125259399, 0.22075416147708893, -0.011260581202805042, -2.0224149227142334, -0.12933912873268127, -0.1868780255317688, -1.5324273109436035, 1.3066980838775635, -0.4551980793476105, 0.2739753723144531, -0.6171608567237854, 0.10655640065670013, 2.1790897846221924, -1.8280341625213623, 1.630547046661377, -1.1697697639465332, 0.32215315103530884, -0.39417386054992676, -0.6139330267906189, 1.8486111164093018, 1.4812318086624146, -0.8281736373901367, -0.28755250573158264, 0.10292612761259079, 1.0748087167739868, -0.6708448529243469, 0.6186659336090088, -0.7839388251304626, 0.9087141752243042, 1.0849838256835938, 0.7437752485275269, 0.30117279291152954, 0.806376576423645, 0.26546722650527954, 0.4390316307544708, 0.5159204602241516, -1.0287015438079834, -0.6304745078086853, -0.1610730141401291, -1.5672279596328735, -0.4439002275466919, 1.747262716293335, -0.4694738984107971, -2.8777661323547363, -0.4919551610946655, -0.06811081618070602, -0.2898210287094116, 1.234588861465454, 1.1882067918777466, 0.03609437122941017, 0.3650822639465332, -0.18700189888477325, 0.9338075518608093, 0.7201458811759949, -0.14992757141590118, 0.8587658405303955, -1.0060374736785889, 1.01389479637146, -1.740726113319397, 0.746010422706604, -0.2971365451812744, 1.0262939929962158, -0.5163828730583191, 0.2414618879556656, -0.15790672600269318, -1.7988896369934082, 1.5094716548919678, -1.3170077800750732, -1.3563344478607178, 0.10830140113830566, 0.8741939663887024, 0.49942851066589355, 0.7750981450080872, -2.191708564758301, 0.0819316878914833, 0.8124198317527771, -0.49789735674858093, -1.1014130115509033, 1.228297472000122, -1.8876208066940308, 1.8367505073547363, -0.5596379637718201, 0.7486982345581055, 0.5197233557701111, -1.2838152647018433, 0.3818095326423645, -0.771884560585022, -1.3670496940612793, 0.5720497369766235, 0.1308874785900116, -1.1377696990966797, -1.1216992139816284, 0.2240322083234787, 1.173746943473816, 0.5409035682678223, 0.6480850577354431, -0.8419922590255737, -0.9840995073318481, 1.2803047895431519, 0.006964665837585926, 1.655226230621338, 0.04111664369702339, 0.18627245724201202, 1.0624531507492065, -0.5576564073562622, -0.10963188856840134, -0.7267377972602844, 0.8327034115791321, 1.4364938735961914, -0.23271140456199646, 0.9742130041122437, 2.0213587284088135, -0.1969015747308731, 0.6603798270225525, 0.0194765105843544, 1.6479359865188599, -0.4272462725639343, -1.024628758430481, -2.5572283267974854, 0.538491427898407, -1.5395585298538208, 1.6722803115844727, -1.07144296169281, -1.2523945569992065, -1.224447250366211, -0.26950275897979736, -0.9274826645851135, -0.17303110659122467, -2.071617841720581, 0.9286026954650879, 0.7254250049591064, -2.4226434230804443, -0.0362764336168766, 0.3152053654193878, -0.5104590058326721, 1.4933762550354004, -0.6725507378578186, 0.20645475387573242, 0.21495509147644043, 0.29491323232650757, -0.29515329003334045, -0.6961603164672852, -0.4384007751941681, 0.7667175531387329, 2.2374143600463867, 0.2081514149904251, -0.07108383625745773, -1.2433288097381592, -0.7847731709480286, -0.8704869151115417, -1.4664552211761475, -1.0475302934646606, -0.21877826750278473, 0.7152948975563049, 0.7371054887771606, -0.5593260526657104, 0.2704571485519409, 0.23210561275482178, 0.7707074284553528, 2.064830780029297, 0.08892574906349182, 0.2360811084508896, 2.106858491897583, 0.7455359697341919, 0.4892178773880005, -0.32247427105903625, 2.3050389289855957, -1.9024529457092285, 0.2104702591896057, 1.7778865098953247, -0.19389118254184723, 0.1100601926445961, -0.7144177556037903, -1.3168619871139526, 0.14278428256511688, -0.038568682968616486, -0.33570238947868347, -0.8087047934532166, 0.6393296122550964, 0.1183549240231514, -1.5038498640060425, -0.4911644160747528, 2.9768314361572266, 1.89304518699646, -1.3511035442352295, 1.0866514444351196, -0.1146043911576271, 1.1335484981536865, 0.7573902010917664, 0.3290363848209381, 2.266392230987549, 1.1642292737960815, 0.8784107565879822, 0.028420547023415565, 0.539770245552063, -1.2347133159637451, 0.4920836389064789, 1.5344077348709106, -0.6259896159172058, -0.2803238332271576, -1.285369873046875, 0.16989272832870483, 0.532447099685669, -0.6720432639122009, -0.12436835467815399, -0.9659331440925598, 0.39898037910461426, 1.350600242614746, 1.4186140298843384, 0.9313321113586426, 1.670468807220459, 0.5215999484062195, 1.0785225629806519, -1.8873944282531738, -0.41221076250076294, -2.7522268295288086, -0.10333383828401566, 0.4336896538734436, -0.801946759223938, 1.9552953243255615, -1.4486192464828491, 1.0307117700576782, 0.08018949627876282, -0.03170844539999962, -2.8306851387023926, 0.330870658159256, -0.12092314660549164, -0.00014462636318057775, -1.2350455522537231, 1.931011438369751, 1.7958301305770874, 0.5135225057601929, -0.42945823073387146, -1.1397027969360352, 0.9538466334342957, -0.25771409273147583, -1.2897446155548096, 0.4825529456138611, -0.47319528460502625, -1.3662482500076294, 1.7264559268951416, 0.549765944480896, 0.9926227927207947, 0.27309390902519226, -0.3027769923210144, 0.18057115375995636, 0.1778232604265213, -2.6753180027008057, -0.0820702463388443, -1.5490624904632568, -0.31908804178237915, -0.27809247374534607, -0.2697465419769287, 0.564254105091095, 0.3084305226802826, -0.2175672948360443, -0.1647355854511261, 0.8094258308410645, -0.017639169469475746, 1.360568642616272, -0.31861233711242676, 0.9522694945335388, 0.6408143043518066, -0.8676462769508362, 1.1603941917419434, 0.007619072217494249, 0.014325524680316448, -1.5290791988372803, -2.147294044494629, 0.9551010727882385, -0.45105019211769104, -1.8926708698272705, 1.6921354532241821, 1.5401958227157593, 1.3250677585601807, 0.9812400341033936, -0.1108236014842987, -0.0035619125701487064, -0.8988396525382996, -0.6882986426353455, -0.8815187811851501, -0.8084682822227478, -0.31635284423828125, -0.7168229222297668, 0.8901710510253906, -0.7110836505889893, 0.1933150589466095, 1.1337401866912842, -1.1267619132995605, -0.528472363948822, -0.15915970504283905, -0.4170258343219757, -0.2611585557460785, -1.702418565750122, -1.6238422393798828, 0.028267912566661835, 0.7393274307250977, 0.12582917511463165, 1.7691842317581177, 0.614782989025116, 0.2749417722225189, -0.5333570837974548, -0.5751492381095886, 1.6897660493850708, 0.09261514246463776, 0.260387659072876, -0.6004160642623901, 1.045526146888733, -1.280847191810608, -0.254883736371994, 0.26930469274520874, 0.539765477180481, -0.21943150460720062, -0.3010846972465515, 1.9070769548416138, -1.205316185951233, 0.06715834140777588, -0.3205588161945343, -0.22967877984046936, -0.11975424736738205, -1.0656315088272095, -0.654114842414856, 0.15163378417491913, -0.14224468171596527, 1.5022271871566772, 0.3971322476863861, -0.4028560519218445, 0.6646987199783325, -1.8119226694107056, -0.4635475277900696, -0.3840027153491974, 0.9003220200538635, -0.8725082278251648, -1.255987524986267, 0.7243679761886597, -0.6792246103286743, 1.062782883644104, 0.6774572730064392, -0.3758254051208496, 0.07787960022687912, 0.15678958594799042, 0.2732560336589813, 0.09391732513904572, -0.23401209712028503, 0.7406831383705139, -0.48210757970809937, -0.23379582166671753, -0.8870775699615479, -0.013364088721573353, -1.1547695398330688, 0.6368356943130493, -1.0338630676269531, 0.5363763570785522, -0.5487121343612671, 1.1805795431137085, -1.9322211742401123, -0.7488451600074768, 0.8324106931686401, -0.9761624336242676, 0.12148111313581467, 1.52373468875885, -1.0069793462753296, 1.059998869895935, -0.02426406927406788, -1.084812045097351, 0.04429426044225693, -0.6904191374778748, 0.7705132961273193, 0.20898883044719696, -0.4119231700897217, 0.4230767488479614, -0.46879997849464417, 0.1465514600276947, -1.5915576219558716, 0.19065237045288086, -0.46413248777389526, -0.4450914263725281, -0.5719687938690186, 0.050503235310316086, -1.481007695198059, 0.4764600396156311, -0.42211923003196716, -0.9999479651451111, -0.08114293962717056, -0.19636298716068268, -1.0192885398864746, 1.1321665048599243, 0.9869173169136047, 0.869168221950531, -0.04796595871448517, 0.45174816250801086, 0.38371580839157104, -0.7019683718681335, -1.223829746246338, -0.8318039178848267, 0.5846141576766968, 0.7349967956542969, -2.0393035411834717, -0.08237261325120926, 0.15866155922412872, 0.09547431021928787, -0.24762654304504395, -0.22387054562568665, 0.18461179733276367, 0.4813469648361206, 0.24957706034183502, -0.2526792585849762, -1.643545389175415, -0.943793535232544, 0.4626716077327728, 0.27327659726142883, -0.6105985641479492, -0.03796723857522011, -0.9918960332870483, -0.8154920339584351, 0.27445167303085327, -0.7996312379837036, 0.1361236274242401, -0.5450251698493958, 0.19772085547447205, -0.6411648988723755, -0.11307664960622787, 0.85658860206604, 0.6322959661483765, 1.0702100992202759, -0.5642879009246826, 1.4900764226913452, -2.272785186767578, 0.483027845621109, 0.4001266360282898, -0.15920840203762054, 0.12378589063882828, 1.332948923110962, 0.21692752838134766, -1.864089846611023, -1.679521083831787, 0.26861095428466797, 0.07581736892461777, 1.031235694885254, 1.042493462562561, -0.7036252021789551, 0.13350552320480347, -0.6163219213485718, -1.632167100906372, 1.7478796243667603, -0.28077423572540283, -1.0510565042495728, 0.16623076796531677, -0.6675294041633606, -0.40903109312057495, -0.24855968356132507, -0.9437270760536194, 0.7736085057258606, 0.8845680952072144, 0.19337059557437897, -0.024471350014209747, -1.2535244226455688, -1.3016573190689087, -0.3052828013896942, -0.3403223156929016, 0.44841331243515015, 1.0158343315124512, 0.5314012169837952, -1.0585250854492188, -0.46898913383483887, -1.6415317058563232, 0.40780556201934814, -0.09128612279891968, -0.7005414366722107, -1.5934193134307861, 0.6244854927062988, -0.5329745411872864, 1.095001220703125, -0.07879482954740524, -0.8746573328971863, 1.1203645467758179, 0.7353000640869141, -0.376417875289917, 0.461577445268631, -0.34829622507095337, 1.5817158222198486, -0.8642874360084534, 0.6093682050704956, 2.3186426162719727, -0.018953250721096992, -0.08526669442653656, -0.5928621888160706, -0.05990318953990936, -0.5152251124382019, -1.2084957361221313, 0.5902127027511597, -0.4911394715309143, -0.5294517278671265, 1.818290114402771, 0.7174760103225708, 0.13043317198753357, 0.9332682490348816, -1.6331512928009033, 0.12060482800006866, -1.3338207006454468, -1.9819228649139404, 2.2272961139678955, -1.1646416187286377, -0.5839595794677734, -1.0359164476394653, 0.4584190249443054, -0.8722549676895142, -1.590388536453247, 0.2177819162607193, -0.03858884051442146, 0.5725283026695251, 0.8995903730392456, 0.646775484085083, -0.4564206898212433, 1.3585971593856812, 1.0031665563583374, -0.22148142755031586, -0.7836183309555054, 0.8140074610710144, 0.7682858109474182, 0.6048680543899536, 1.292799949645996, -0.29702112078666687, -0.5927672386169434], [-1.0298393964767456, 0.19265371561050415, -0.08838547021150589, 0.9782926440238953, -0.2646145522594452, 0.8652474284172058, 0.4637886881828308, 0.6406242251396179, 0.7731316089630127, -0.7408772110939026, -0.35483986139297485, 0.9426040649414062, -0.30865082144737244, 0.343531996011734, 0.6869216561317444, 0.6353254914283752, -0.26058363914489746, -2.4736955165863037, -1.3873916864395142, -0.9994311928749084, 0.5141211152076721, 2.3423287868499756, 0.008641990832984447, -1.258651852607727, 0.5112609267234802, -0.061694126576185226, 0.3558807671070099, -0.25552505254745483, -0.4123925268650055, 1.4267598390579224, 0.039233312010765076, -0.6962464451789856, -1.0382627248764038, 0.06233881041407585, 0.27537843585014343, -1.2244325876235962, -1.0349222421646118, -0.15656712651252747, 0.5412868857383728, 0.5314552187919617, -1.8147923946380615, -0.02506871707737446, 2.4496681690216064, -0.042420558631420135, -1.4720913171768188, 0.15285061299800873, -1.084905743598938, -2.0263519287109375, 0.04441097006201744, 1.7805525064468384, -1.755927324295044, 0.2808435261249542, 0.07864931225776672, 0.3540748357772827, -2.1442344188690186, 1.3432259559631348, -1.777626872062683, -2.463942766189575, 0.11570609360933304, 0.2027520090341568, -0.26361083984375, -2.6329798698425293, -1.2859642505645752, -0.07364775240421295, 0.006821662187576294, -0.6648054718971252, -0.17041613161563873, 1.310454249382019, 1.2419216632843018, -1.4383783340454102, 1.396448016166687, -0.21675042808055878, 1.3302797079086304, 0.9687029123306274, 0.30896005034446716, -0.31606918573379517, -0.06240302696824074, -1.3870458602905273, -2.0518157482147217, 0.4767366349697113, -0.42662227153778076, -1.144113540649414, -0.8771156072616577, 0.1836683750152588, 1.9021581411361694, -2.849919319152832, 0.6262339949607849, 2.4858226776123047, -0.235421821475029, 0.8612288236618042, 0.5396496653556824, -1.3219696283340454, 1.5823568105697632, -1.3250625133514404, -0.6698535084724426, -0.8613107204437256, -0.4728221595287323, -3.0196902751922607, 0.6214029788970947, 0.22064870595932007, -0.8368370532989502, -0.022391580045223236, -0.5089715123176575, 1.004520058631897, 0.4691472351551056, -0.7018998265266418, -0.08219427615404129, -1.252936840057373, -2.3540878295898438, 0.2033679336309433, 1.024289608001709, 0.7129506468772888, 1.0012301206588745, 0.15315654873847961, 1.228434681892395, -0.7139829397201538, -1.6522438526153564, -0.7584803104400635, -0.6518793106079102, 0.6244445443153381, 0.8377028703689575, 1.7577221393585205, 0.3250971734523773, 0.7033412456512451, 0.09842263907194138, -0.9466173052787781, -0.3579345643520355, -1.598554253578186, 0.8610947132110596, 1.9642224311828613, -0.24841201305389404, 0.32804977893829346, 1.5671347379684448, 0.692541778087616, 0.8691245913505554, -1.114043116569519, 1.5493403673171997, 0.7307687401771545, 0.7686424851417542, 1.7953466176986694, 0.9190050363540649, -1.8271467685699463, 1.1510910987854004, -1.3503918647766113, -0.4792485237121582, -1.439826488494873, 1.014718770980835, -0.7658326625823975, 0.8283209800720215, -1.8743059635162354, -0.5832124948501587, 0.4455653429031372, 0.1718088537454605, -0.3835163414478302, -1.2340713739395142, 1.053716778755188, -0.8241705894470215, -0.7172545194625854, 1.515257716178894, 0.4350586533546448, -1.6033087968826294, 1.2383041381835938, -0.6727229952812195, 1.7941246032714844, -0.4622804522514343, 1.1445305347442627, 1.4786713123321533, -0.977828323841095, -0.0016693677753210068, 1.8845267295837402, 0.3525870442390442, -0.36330941319465637, 2.4634714126586914, -1.787468433380127, 0.04129921272397041, -0.704742968082428, 0.9950141906738281, -0.6093722581863403, -0.8186518549919128, -0.09752865880727768, -2.128258228302002, -1.0247454643249512, 0.2648194432258606, -0.19182583689689636, 0.03730358928442001, 0.3688495457172394, -0.7810898423194885, 0.39476072788238525, 0.3066626787185669, -0.2674061357975006, -0.26374194025993347, 0.31434133648872375, -1.6999478340148926, -2.389136791229248, -0.39963075518608093, 1.4162763357162476, 1.0862815380096436, 0.5468953251838684, 1.9078627824783325, 0.6118494868278503, 0.05894375219941139, -0.4655102789402008, -0.1336071789264679, -0.5371764302253723, -0.9758719801902771, -0.07748603075742722, -1.2075722217559814, -0.7541323304176331, -1.6608647108078003, 0.42424657940864563, 0.44034498929977417, -0.3850438892841339, 1.0362178087234497, -0.47578296065330505, 1.4110596179962158, -1.0731854438781738, -0.5234042406082153, 1.7879669666290283, -1.8344969749450684, -0.7703131437301636, -1.0086791515350342, -0.9590258598327637, 0.3891773223876953, 0.005295260343700647, -1.3877546787261963, -1.0884158611297607, -0.14993077516555786, -0.9093614816665649, -0.1584538370370865, -0.7772931456565857, 0.5345896482467651, -1.5147162675857544, -0.7861294150352478, 1.6270747184753418, 1.0819871425628662, -1.418206810951233, 1.0493216514587402, -0.4144602417945862, 1.3496538400650024, -0.6170807480812073, 2.5612058639526367, 0.2786588668823242, -0.5616324543952942, 1.298371434211731, 0.24557451903820038, 2.532167673110962, -0.01921951398253441, 0.8573598265647888, -1.4626355171203613, 0.1563710868358612, -0.7108087539672852, -0.7282769083976746, -1.1394094228744507, -0.15242542326450348, -0.43220311403274536, 0.24027583003044128, 0.6482868790626526, 1.3593724966049194, 1.8899227380752563, -0.07686594128608704, -0.5431387424468994, 1.984553575515747, 0.7806720733642578, -0.05997220426797867, -0.11538060009479523, 1.3975703716278076, -2.104388952255249, -0.8421702980995178, 1.6297779083251953, 0.5863268971443176, 0.2386569231748581, 1.0511595010757446, 0.060428209602832794, -0.9782498478889465, -1.0278788805007935, 0.7998663187026978, -1.4750219583511353, 0.9899506568908691, 0.4143393039703369, -0.6850833296775818, 0.01663784123957157, 0.15343452990055084, -0.4885508418083191, 0.8238565921783447, 0.014735792763531208, -0.1636921614408493, -0.7590312361717224, -0.11064472794532776, -0.9158819317817688, 1.1080578565597534, 0.8842021226882935, 1.429734706878662, -0.3112016022205353, -0.2006969302892685, -0.3942714035511017, 0.3109574317932129, 0.7359793186187744, -1.005759596824646, -1.6812381744384766, -1.239811897277832, -1.335152506828308, 0.5502992868423462, 1.5881547927856445, -0.8015552163124084, 1.2951163053512573, -1.0396603345870972, 0.7461956739425659, 0.7927156090736389, 1.2119327783584595, 1.346913456916809, 1.0013291835784912, -0.8189138174057007, -0.1334547996520996, -0.014416089281439781, 0.3154403269290924, 0.3954451382160187, 1.8361294269561768, -0.8739588260650635, -1.4481920003890991, 2.368945598602295, 1.286346197128296, 0.06303314864635468, -0.10138077288866043, -1.9125937223434448, -0.5023097991943359, -0.8256587982177734, 0.22377464175224304, -0.3092692494392395, 0.8540271520614624, 0.4150928258895874, -0.6671881675720215, 0.6780509352684021, -0.7869136333465576, -0.915888249874115, 0.08805204927921295, 0.19063203036785126, 0.10289844125509262, -0.48255598545074463, -0.3527921140193939, -0.9464932084083557, 2.4027321338653564, 0.3577965199947357, 0.6004445552825928, 0.17858842015266418, 1.0574119091033936, -0.48122385144233704, -1.5810366868972778, -0.5637950897216797, -1.3731677532196045, -0.673007607460022, 1.9688044786453247, 1.6123813390731812, -0.08757337182760239, 1.2402235269546509, -1.183598279953003, -0.09207315742969513, -0.471614271402359, -1.7279250621795654, -1.1522173881530762, -1.2950019836425781, -0.3480975329875946, -0.7220428586006165, -1.2387149333953857, -0.700217604637146, 1.2001460790634155, 0.8778765201568604, 0.323154479265213, -0.9904327392578125, -1.4202708005905151, -0.5823177099227905, 0.06772361695766449, -0.2774576246738434, -1.07539963722229, 0.531924307346344, -0.42960283160209656, 1.084050178527832, -1.3669326305389404, 0.25897154211997986, -0.056620899587869644, -0.36779725551605225, 0.7508045434951782, -0.10185885429382324, 1.9622769355773926, -0.369605153799057, -0.8710348606109619, 0.4885013699531555, -0.17347843945026398, -0.7460053563117981, -1.2747329473495483, 0.44746506214141846, 1.1186468601226807, 0.11016394942998886, 1.6585731506347656, 2.4247474670410156, 0.8220717310905457, -1.6527597904205322, -0.4082150459289551, 0.39565229415893555, 0.4218669831752777, -1.211530327796936, -0.7480500340461731, -0.039332374930381775, 0.6665928959846497, -0.10506050288677216, -0.0263716671615839, 0.20258945226669312, 0.09658826887607574, -0.5306437015533447, 0.38037824630737305, 0.09969820082187653, 0.36209550499916077, -1.059112787246704, -0.11172162741422653, -0.4790363311767578, 0.033990707248449326, 1.4832780361175537, -1.1833058595657349, 0.3410508334636688, 0.4022313356399536, -0.10673145949840546, -1.7556724548339844, 1.5005460977554321, -0.9640134572982788, 0.20735366642475128, 0.6435527205467224, -1.7707164287567139, 0.8029521107673645, 1.5994521379470825, 0.09070248156785965, 0.31613442301750183, 0.2400222271680832, 0.3284848928451538, -0.13423113524913788, -0.998253345489502, -0.27405357360839844, -0.33913102746009827, -0.8653085231781006, -0.8627123832702637, 0.024012921378016472, 0.25775712728500366, -0.7464255094528198, -0.44882073998451233, 0.29008349776268005, -0.6120432019233704, 0.39362528920173645, 0.590477705001831, -0.1455688625574112, 0.241660937666893, -0.8181326389312744, -0.2614935338497162, 0.19447393715381622, -0.5328343510627747, 0.20433151721954346, -0.03385792300105095, -1.0799224376678467, -0.45759695768356323, 0.8015109896659851, -0.8634567260742188, 0.17305321991443634, -0.4786326289176941, -0.1453876942396164, 1.231146216392517, -1.1686124801635742, 0.05415850132703781, -2.8936493396759033, -1.0814019441604614, 1.0287690162658691, -0.0990590751171112, 1.3444023132324219, -0.4986361265182495, -0.8034200668334961, 0.2183002531528473, -0.4027549624443054, -0.3404977023601532, 0.9912110567092896, 0.016612965613603592, 0.08589716255664825, -1.9447038173675537, 0.3672403395175934, -0.18232044577598572, -0.5061320066452026, 0.18140952289104462, -1.732762098312378, 0.06813068687915802, 0.2656213641166687, -0.2613404095172882, 1.9471609592437744, 0.3920425474643707, 1.8808256387710571, -1.9382113218307495, -0.4559922218322754, -1.063872218132019, -0.3120545446872711, -0.6633410453796387, -0.5624900460243225, 0.23538237810134888, 0.43308594822883606, 1.2860549688339233, 0.7661859393119812, -1.0635714530944824, -0.039224009960889816, 1.2954684495925903, 0.26006314158439636, 1.2685716152191162, 0.29677614569664, -0.16957244277000427, -0.08372776210308075, 1.4951145648956299, -0.0911720022559166, 1.096062183380127, -0.6793456077575684, -0.29777905344963074, -0.4029937982559204, 0.09394678473472595, -0.5784684419631958, 0.18031533062458038, 0.5620064735412598, -0.08625742048025131, 0.5278862714767456, 0.29855045676231384, -1.407599687576294, -0.6271436214447021, 0.9080492258071899, -0.44729194045066833, 1.0898224115371704, -1.7053892612457275, -0.49922463297843933, -1.186629056930542, 0.015030691400170326, -1.0455669164657593, -0.30458226799964905, 0.09639820456504822, -0.6833966374397278, -0.6078844666481018, -1.0670299530029297, 0.886177122592926, 1.463244080543518, 0.5411291718482971, 0.6959751844406128, 1.1948764324188232, -0.8733218908309937, 1.137737512588501, -0.22597722709178925, 1.0366922616958618, -0.20930276811122894, 1.8372117280960083, 0.4471326768398285, -0.8068768382072449, 0.21354839205741882, -0.7855357527732849, 0.69297194480896, -1.4405579566955566, 0.7112370729446411, 2.05322003364563, -0.26634681224823, 0.11313820630311966, 0.1372145414352417, 1.5732696056365967, 0.35829684138298035, 0.14967669546604156, 1.8592476844787598, 1.3208283185958862, 0.40412071347236633, 0.3115650415420532, 1.5044664144515991, 0.8158198595046997, -0.13937655091285706, -0.15289489924907684, 1.4701217412948608, 0.7567298412322998, -0.9544243216514587, 1.3670510053634644, 0.20925086736679077, -0.020715266466140747, -2.5226268768310547, 0.8967999815940857, 0.24379335343837738, -0.922588586807251, 1.4041047096252441, -0.7386197447776794, 0.11142542213201523, -0.6992200016975403, -0.0008990505011752248, -1.7830487489700317, -0.4039442539215088, -0.8044403791427612, 0.5781105160713196, -0.9034880995750427, 1.7090409994125366, -1.801239252090454, -0.6931983232498169, 1.4747453927993774, 0.3915092349052429, -0.5282816886901855, -0.268266886472702, 0.525880753993988, 0.7992345094680786, -1.2979354858398438, 0.9726505875587463, -0.7468381524085999, -0.28724825382232666, 0.7410798668861389, -0.17585192620754242, -2.5457358360290527, -0.44650357961654663, 0.4086807072162628, -2.4874846935272217, -0.06503160297870636, -0.14745213091373444, 0.19746612012386322, 0.44766730070114136, -1.3413296937942505, 1.2866344451904297, 0.6113160848617554, -0.27548524737358093, -0.5207479596138, 0.7417012453079224, 1.4562832117080688, -0.9165996313095093, 0.7268111109733582, -0.5469620823860168, 0.18640395998954773, -0.7881553173065186, -0.40241530537605286, -0.7379931211471558, -1.4396207332611084, 0.675112247467041, -0.2557566165924072, 0.2895212471485138, -0.6111607551574707, 1.9600350856781006, -0.017796438187360764, 1.1870338916778564, 1.448861002922058, 1.5759438276290894, 1.0590226650238037, -0.7255614399909973, -0.36920422315597534, -0.9837478995323181, -0.026337048038840294, 2.0701842308044434, 0.4668044447898865, -0.7638094425201416, 0.9842451214790344, 1.0718791484832764, -0.8982158303260803, 0.809013307094574, 0.3896790146827698, 0.362289696931839, 1.1547821760177612, 1.3333475589752197, 1.2625510692596436, 0.8421403169631958, 0.8896470069885254, 0.4259975552558899, -1.545436978340149, 2.671832323074341, 0.16607797145843506, 0.045315586030483246, -1.6240853071212769, -1.0119167566299438, -1.0448541641235352, -1.2327203750610352, 0.05580335482954979, -0.7628138065338135, 0.07994742691516876, 0.4978120028972626, 0.519334077835083, -0.5821353793144226, -0.1979481428861618, 0.6361700296401978, 0.07439819723367691, -0.1300153285264969, 0.22531898319721222, 0.3967585861682892, -0.10850591212511063, 0.3450855016708374, -1.0115503072738647, -0.16027289628982544, -0.05504714697599411, -0.2681901752948761, -1.0435998439788818, -1.8794907331466675, 0.4251794219017029, -1.176002860069275, -0.11788789927959442, -0.4558376669883728, -0.40133124589920044, 0.1456775814294815, -2.0728142261505127, -0.7768549919128418, -0.3926122188568115, -1.3076328039169312, 1.1160290241241455, 0.13706792891025543, 1.3638767004013062, -1.7785561084747314, -1.3563252687454224, 0.23875705897808075, 1.420994520187378, -1.1200833320617676, 0.11699974536895752, 1.0505656003952026, -0.6955481767654419, 0.40896540880203247, -0.5679675340652466, 0.018601538613438606, -0.18764492869377136, -0.978325605392456, 0.00544415507465601, -0.40895962715148926, 0.7764151692390442, 0.22471491992473602, -0.6890729665756226, -0.08531730622053146, 0.7486971020698547, 0.31863459944725037, 0.6507445573806763, 0.7286654114723206, 0.30033063888549805, 0.8542919754981995, -1.5752465724945068, -0.5844337940216064, -1.0111163854599, 2.1717066764831543, -0.2493581473827362, -1.6465731859207153, 0.09510812908411026, 0.48960790038108826, 0.3095948100090027, 1.1143708229064941, -0.6044901609420776, -0.06939368695020676, 0.8092712759971619, 0.81880122423172, -0.8282620906829834, -1.6552735567092896, 0.1319088190793991, -0.03893345594406128, -2.026374578475952, -0.5896854400634766, -0.2325126826763153, -0.06963817030191422, -0.09420467913150787, 0.30449482798576355, 0.6715935468673706, 1.9842146635055542, -1.3841811418533325, 1.6793489456176758, 0.8821710348129272, 0.5831347703933716, 1.3558889627456665, 0.43679118156433105, 0.7486091256141663, 0.7503325343132019, 0.9519212245941162, -0.6032193303108215, 0.7066656947135925, -0.5021479725837708, 0.8737351894378662, 2.8350279331207275, 0.6342974305152893, -1.1724580526351929, -0.06814317405223846, 0.3516126871109009, 1.3823968172073364, -0.4861071705818176, 1.5314587354660034, -0.6719903349876404, -0.5518023371696472, -1.3162009716033936, -0.9595006704330444, -0.038382306694984436, -0.8034635186195374, 0.6459068655967712, 0.2781750559806824, 2.043538808822632, -1.6946697235107422, 2.058115243911743, -1.0928508043289185, 0.09940198808908463, -1.1306277513504028, -0.4274718463420868, 0.6938194632530212, 1.6418170928955078, 1.381137490272522, 0.5341973304748535, -0.35111841559410095, 1.3494751453399658, 0.43907755613327026, 0.41307899355888367, -0.31867754459381104, 1.1439136266708374, -0.16679668426513672, -1.0875247716903687, -0.8698345422744751, 0.11709653586149216, 0.029265984892845154, -1.1699706315994263, -1.3555899858474731, 0.5743721723556519, 0.7887387275695801, 2.0136189460754395, 0.18239620327949524, 0.7426860332489014, -0.9561329483985901, -0.8871033191680908, -0.3702889382839203, 0.7342762351036072, -0.08141056448221207, -2.381147623062134, 0.46934974193573, -1.4685910940170288, 0.6617133021354675, -0.7525597214698792, -0.49739742279052734, -0.7716575860977173, 0.8031657338142395, 0.44208237528800964, 1.6192843914031982, -0.8769475817680359, -0.02827560529112816, -1.070611834526062, 0.028608690947294235, -1.0625157356262207, -0.405769944190979, 1.662285566329956, 0.28108713030815125, 1.313592553138733, -0.2854268550872803, -0.5687316060066223, -0.8536615967750549, 1.14315664768219, 0.41464680433273315, -0.23638051748275757, 0.404603511095047, -0.7121208906173706, -2.096238136291504, -0.9966159462928772, -0.18909692764282227, 1.136147379875183, -0.15402334928512573, 0.1349344551563263, -0.4857214391231537, -1.7793127298355103, 0.6285459995269775, -1.2540134191513062, 1.0421967506408691, 0.09617892652750015, 0.6384183168411255, 2.2436392307281494, 0.7238507270812988, 1.5405292510986328, 0.4121425449848175, -0.3603264093399048, 0.6093267798423767, -0.6383455991744995, 1.6754710674285889, 0.21948476135730743, -1.1913988590240479, -0.6472952365875244, -0.3987155556678772, -0.5708633065223694, -0.05431355908513069, -0.2643119990825653, 0.3486081063747406, 0.7289019227027893, -0.5832144618034363, 1.293055534362793, -0.14643922448158264, -0.19828301668167114, 0.38293424248695374, 0.5156791806221008, -0.017180169001221657, -1.512518286705017, 1.9722267389297485, 1.3047230243682861, 0.30188316106796265, 0.2848978340625763, 0.7779855728149414, -1.374755620956421, 1.1296663284301758, 1.062735915184021, -1.1298649311065674, -0.38773423433303833, -0.2623487114906311, -0.32875558733940125, -0.5526678562164307, -0.5314643979072571, -0.6420848965644836, -0.6894785165786743, 0.7771345973014832, 1.1395745277404785, -1.06925368309021, -0.018412675708532333, 0.706526517868042, -2.366137981414795, -0.2821693420410156, 0.1208191066980362, 0.7296575307846069, -0.557603657245636, 0.32691797614097595, 0.6071186661720276, -1.1484522819519043, -1.161612868309021, 1.8483725786209106, 0.07498906552791595, -0.37719976902008057, -0.45394712686538696, -2.014181137084961, -0.05361232906579971, 0.29256659746170044, -0.06205711141228676, 1.7767070531845093, 0.21413759887218475, -1.360482931137085, -0.6219045519828796, 1.0141276121139526, -0.6884713172912598, -1.5135858058929443, -0.02125105820596218, 2.527230978012085, -1.4782167673110962, 0.13004405796527863, 0.38717877864837646, -0.23055113852024078, -0.8785211443901062, -1.1674376726150513, -0.9208532571792603, 0.5040059089660645, 1.2484289407730103, -0.40356379747390747, 1.782971978187561, -1.7264379262924194, -0.9979240298271179, 0.2218252718448639, 1.962416172027588, 1.216251015663147, -1.3910391330718994, 0.9921862483024597, 0.9085927605628967, -1.4613655805587769, 0.19126132130622864, -0.296991229057312, 1.12763512134552, -0.6036009788513184, 0.570631206035614, -1.298710823059082, 1.3981600999832153, -0.023410802707076073, -0.9672950506210327, 0.9896302819252014, 0.32700249552726746, 0.5442487001419067, 0.28173449635505676, -0.9249439835548401, -1.9615960121154785, 1.157618761062622, 0.2860267162322998, 0.425801157951355, -0.5366536378860474, 0.7069145441055298, -0.30569520592689514, -1.9019947052001953, 1.429160475730896, 0.13042491674423218, 0.7879288196563721, -0.4931773841381073, -0.344490647315979, -0.30356740951538086, -0.2062007486820221, 0.6890678405761719, -0.04932072386145592, 2.0012083053588867, -0.983388364315033, 0.48317623138427734, -0.34050700068473816, -0.05750555172562599, -0.6241545081138611, -1.0838618278503418, 1.8573780059814453, -2.8806264400482178, 0.34690943360328674, 1.1818890571594238, 2.1460704803466797, 1.5111526250839233, -0.44279149174690247, -0.6220341324806213, -0.47787365317344666, -0.9916148781776428, -0.3008768856525421, -0.04545921832323074, -0.19988438487052917, -0.4240177273750305, -0.4038485586643219, 0.3365151584148407, -1.3943864107131958, -0.2803594470024109, 0.28731316328048706, -0.5857291221618652, -1.491980791091919, 0.18077948689460754, 0.36427175998687744, -0.3815721571445465, -1.5133378505706787, 0.8141483664512634, -0.2944891154766083, -0.5652429461479187, -0.9593259692192078, -1.0063254833221436, 1.1093441247940063, 0.821101725101471, 0.030839847400784492, -1.1048154830932617, 0.2989301383495331, 1.0007166862487793, -0.798558235168457, -0.22610712051391602, -0.5031214356422424, 0.15061289072036743, 0.08809685707092285, 2.608816146850586, 0.7852828502655029, -0.8232558369636536], [0.6760060787200928, 1.3243898153305054, 0.9418275356292725, 0.23366199433803558, -0.3591594994068146, 1.2744989395141602, 2.0659844875335693, -0.33375900983810425, -0.6832613945007324, 1.2089811563491821, -0.3515506386756897, -2.241624593734741, 1.2691235542297363, -0.13776461780071259, 0.5870348215103149, 0.99220210313797, -1.144990086555481, -0.12380697578191757, 1.1784226894378662, 1.32286536693573, 1.7950247526168823, -0.976954996585846, 1.616357445716858, 0.9736179113388062, 0.31081199645996094, 0.6258977055549622, 0.6467800736427307, 1.2169100046157837, -0.4534940719604492, 0.07690926641225815, -0.5093402862548828, -1.176866054534912, 0.9055142402648926, 1.1993242502212524, -0.3628160059452057, -0.38670989871025085, -0.22315560281276703, -1.8025490045547485, 1.1943755149841309, -0.5447427034378052, 1.8051997423171997, 0.371904194355011, -1.0425300598144531, 0.275624543428421, -0.7384523153305054, 0.07149211317300797, 1.3795348405838013, -0.3924812078475952, -0.8037009835243225, -0.062112558633089066, -2.033884286880493, -0.9430544972419739, 0.112680584192276, -0.9498617649078369, 0.3091873824596405, 0.9280071258544922, 0.02822365239262581, -0.4614989757537842, -0.5578498840332031, -0.10264550894498825, 1.709398865699768, 0.20119641721248627, -0.5270546674728394, 0.05567611753940582, 0.45076417922973633, 0.383786141872406, -0.3557598292827606, 2.1687440872192383, 0.6105716228485107, -1.533409595489502, -1.099826693534851, -0.6582397818565369, 1.434138536453247, 1.0848942995071411, 1.4249213933944702, 2.159628391265869, -1.155447244644165, -1.0037559270858765, 0.529198408126831, 0.6426282525062561, 0.5456608533859253, 1.3953956365585327, -0.9885550737380981, -0.6795923709869385, 1.4473925828933716, -0.22730793058872223, 0.549383819103241, -0.07441586256027222, -1.0380922555923462, -0.08766555786132812, -0.20349165797233582, -0.4129312336444855, 0.5987729430198669, 0.01844366267323494, -1.4207757711410522, 1.2170672416687012, 1.1373366117477417, 0.5129304528236389, 0.31509748101234436, 0.14004582166671753, -0.4018573462963104, -0.6128404140472412, 0.19873228669166565, -0.5146149396896362, -2.431861400604248, -1.8713393211364746, 0.08586418628692627, 0.6794824600219727, -0.3943043649196625, -0.7279030680656433, -0.5496751666069031, 0.9288203120231628, -0.9801415801048279, 1.4065536260604858, -1.295839548110962, 0.27128660678863525, 1.2839462757110596, 0.6753774881362915, 0.002297750674188137, -0.7889478802680969, -0.42661231756210327, 0.2990770637989044, -0.4812789559364319, 1.4943941831588745, -1.9408445358276367, 0.6787692308425903, 1.053990125656128, 0.49065348505973816, 0.12022223323583603, 0.15162736177444458, 0.35846075415611267, 1.2344647645950317, -0.10211452841758728, -1.412010669708252, -0.4434523582458496, -1.2787041664123535, 0.7253544926643372, 0.7732802033424377, 1.1950950622558594, -0.5310958027839661, 0.9986315369606018, 1.3531856536865234, -0.06531307846307755, 0.3380674719810486, -0.585066556930542, 0.678386926651001, -0.8091838359832764, -0.889418363571167, -0.747744619846344, -0.9878208637237549, -0.09029204398393631, 1.4956597089767456, -0.2524464726448059, -0.13079147040843964, 2.7051546573638916, -0.7525065541267395, -0.14669489860534668, -0.28023403882980347, 0.3149060010910034, 0.2922663986682892, -0.6042141318321228, -1.192475438117981, -1.0983787775039673, -0.8721356987953186, -0.2680005431175232, 0.7059595584869385, -1.3371059894561768, 1.3805437088012695, 0.10394100844860077, -0.20171689987182617, 1.5851852893829346, 1.2103947401046753, 0.18486866354942322, -0.8857342004776001, -0.7496158480644226, -0.2134122997522354, 0.1360398381948471, 0.306155264377594, 2.4518744945526123, 0.5320962071418762, 0.8599799275398254, 0.05225210264325142, 0.41719284653663635, -0.8476485013961792, 0.9021047949790955, 1.7307204008102417, -1.8938320875167847, 0.5433177947998047, -0.17723257839679718, -2.0504348278045654, -1.8923163414001465, 0.2791311740875244, 1.276832103729248, 1.7528256177902222, -0.38504713773727417, 0.6923056840896606, 0.38196608424186707, -0.5477568507194519, 1.3885133266448975, 0.4884414076805115, 0.4091484546661377, 0.24359939992427826, -1.0054787397384644, 0.9309030175209045, 0.14250092208385468, -1.2359882593154907, 0.5097200274467468, -1.1537132263183594, 0.3012577295303345, 0.3467157483100891, -0.05361520126461983, 0.4468345046043396, 2.0193867683410645, 0.4567578136920929, -0.4576719105243683, 0.659864604473114, 1.0182150602340698, 1.1963027715682983, -0.12398350238800049, 0.39906010031700134, -0.4021055996417999, -0.37170839309692383, -1.222678780555725, -2.5950021743774414, -0.6782230138778687, 0.06136171147227287, 1.3151943683624268, 1.1812692880630493, 0.042856860905885696, 0.9425306916236877, 1.1774801015853882, 0.5650506615638733, -1.1667450666427612, 1.371022343635559, -1.0831385850906372, 1.4590147733688354, 0.4003073275089264, -0.06945830583572388, 0.630905270576477, 0.18727163970470428, 1.309369683265686, -0.23302359879016876, -0.131147563457489, -0.1444944441318512, -0.2940983772277832, 0.4454907476902008, 1.8034696578979492, 1.3121663331985474, 1.3027232885360718, 0.7552344799041748, -0.17968566715717316, 0.1906016319990158, -0.2898382246494293, -1.4388691186904907, 0.37249067425727844, -1.6383124589920044, -0.4292874336242676, -0.5653748512268066, -0.35827022790908813, -1.3783421516418457, -1.008764624595642, 0.4886693060398102, 0.188698410987854, 0.19294385612010956, 1.85148286819458, 1.076117992401123, -0.4161681830883026, 0.29678231477737427, -0.08827586472034454, -0.08032607287168503, -0.5787225961685181, 0.35245636105537415, 0.15800732374191284, 1.188734531402588, 0.7186100482940674, -0.8094882369041443, -2.0170600414276123, 0.8728311657905579, -1.6745591163635254, -0.2541719377040863, 1.4828345775604248, 1.0729012489318848, -0.8695762753486633, 0.6840890049934387, 0.9193040132522583, 0.07638509571552277, 0.7292821407318115, -0.9296925067901611, -1.0086973905563354, 0.07117801904678345, 0.03581072390079498, -0.7103760242462158, 0.11570034921169281, -0.0014743708306923509, -0.576215922832489, -0.25213783979415894, 0.9358237981796265, -0.31280651688575745, 0.9124173521995544, 1.6383053064346313, -0.4452902376651764, 0.2527349591255188, 0.3500831723213196, -2.329277276992798, -0.9376893639564514, 0.17536182701587677, 1.1298891305923462, 1.0851701498031616, -0.5382425785064697, 1.6070806980133057, -1.3310185670852661, -2.521594762802124, -0.544359028339386, 1.2161684036254883, 0.45240670442581177, -0.45800143480300903, -0.04799771308898926, 0.04865362122654915, 0.1189471110701561, 0.8804361820220947, -0.6150485277175903, 0.7825788259506226, -1.1565057039260864, 0.4290452301502228, -0.41563451290130615, 0.144854336977005, -3.321051836013794, 0.5433929562568665, 1.4755679368972778, -1.1319496631622314, 1.4490444660186768, 0.006879462394863367, 0.24300870299339294, 0.0471312515437603, 0.3089083731174469, 0.6494356393814087, -0.034855686128139496, -0.764244794845581, -0.14842258393764496, 0.07794208079576492, 0.498899906873703, -0.7045902013778687, 0.9089798927307129, 1.194470763206482, 0.9400684237480164, -0.7610049843788147, -0.7859065532684326, -0.21695314347743988, -1.011071801185608, -1.5991261005401611, 0.6111856698989868, 2.0132367610931396, 1.150992512702942, 0.8475357294082642, -0.2876184284687042, 2.580080032348633, 0.8763940334320068, 0.34998103976249695, 0.4055873155593872, -1.3084818124771118, -0.0838254913687706, -0.09374260902404785, -0.3817388415336609, 1.784469723701477, -1.1899398565292358, 1.4850226640701294, -2.4339773654937744, -2.6986098289489746, 3.1462419033050537, 0.07639332115650177, 0.914604663848877, 0.21593055129051208, 1.3229765892028809, 0.35515376925468445, 1.3023145198822021, -0.7978177070617676, -0.09513446688652039, -0.9818505048751831, 0.017745181918144226, 0.5795738101005554, 0.5966827869415283, 0.24073264002799988, 1.3238059282302856, -0.32983681559562683, 0.17515213787555695, 0.2413783073425293, 0.9773073196411133, -0.5886130332946777, -0.1108577698469162, 0.04940701648592949, -0.9505718350410461, -0.0902981236577034, -0.44904375076293945, -0.07847093045711517, 0.5364384651184082, 0.9051663875579834, 0.6913347244262695, -0.6512212753295898, -1.5462050437927246, -0.034293949604034424, -1.0388177633285522, 0.31066733598709106, -1.2135157585144043, -0.4892016649246216, 1.053538203239441, -1.4243589639663696, 0.9617777466773987, 0.3218763470649719, -1.6580150127410889, 0.5735430121421814, -0.03449586406350136, -0.16605466604232788, -1.1472423076629639, -0.11706320941448212, 1.4708013534545898, -0.40687915682792664, -0.27273276448249817, 0.41799336671829224, 0.22610458731651306, 0.37270113825798035, 0.4200528562068939, -0.29612475633621216, 0.06841569393873215, 0.3646599054336548, 0.2751459777355194, 0.0837595984339714, 0.6149510145187378, 0.3870004415512085, 0.9707576036453247, 0.5806378126144409, 0.596383810043335, -0.7669574022293091, -0.5809221863746643, 0.56510329246521, -0.5245883464813232, 2.001161575317383, 2.5902793407440186, -2.180738925933838, 0.9904332160949707, 1.5939242839813232, -0.24595250189304352, -0.6679267883300781, 1.2053310871124268, 0.6493920087814331, -2.278149127960205, 1.8201367855072021, 1.610573649406433, 1.4794268608093262, 0.20539617538452148, 0.5854316353797913, 0.4253469705581665, 0.03573411703109741, -0.8301026821136475, 1.9904865026474, 0.03611135855317116, -1.6265759468078613, 0.6164107918739319, 2.0697808265686035, 0.49511364102363586, -0.004501328803598881, 1.0349676609039307, 0.00771741708740592, -0.34564584493637085, 1.9468337297439575, -0.7937686443328857, -1.0760951042175293, -1.1272001266479492, -0.7467671036720276, -0.01674998737871647, 0.10748842358589172, 0.6811555027961731, -0.08582744002342224, -0.4429178535938263, 2.257223606109619, -0.08061077445745468, -0.21039888262748718, 0.18615691363811493, 0.5567994117736816, 0.5044246315956116, -0.48173749446868896, -1.058295726776123, -0.09794031083583832, 0.9498856067657471, 1.870816707611084, -2.9124464988708496, 0.9164182543754578, 0.18531352281570435, 2.216655731201172, -1.2710071802139282, 1.7575182914733887, 0.7323246002197266, 0.032761432230472565, 1.8306715488433838, 2.429722547531128, 0.32867658138275146, -0.4431360363960266, -0.5253089070320129, -1.1596070528030396, 0.9401612877845764, -1.515114188194275, -1.0395879745483398, 0.6078155636787415, -2.2144103050231934, -1.3439805507659912, -0.9627068042755127, 1.1373380422592163, 1.3563860654830933, 0.42009297013282776, 0.21101126074790955, 0.1571745127439499, -0.18845634162425995, -1.2322884798049927, -0.9667271971702576, -0.21355004608631134, -0.5081026554107666, -0.8363122940063477, -0.5996277332305908, -1.1263874769210815, 0.05938360467553139, -0.07606503367424011, -0.3540104329586029, 0.9520192742347717, 1.1030458211898804, -0.8188784718513489, -0.17232000827789307, 0.3940872848033905, 2.0557806491851807, -1.8772236108779907, 0.18817616999149323, 0.8305415511131287, 0.44448769092559814, -0.23631060123443604, -0.4553212523460388, -1.4586237668991089, 0.22647503018379211, 0.5684396624565125, -0.0020561974961310625, 0.6709875464439392, -0.4572734832763672, 2.0680837631225586, 1.0815298557281494, 0.048683978617191315, -2.116245985031128, 0.2799934446811676, -0.3735847771167755, -0.25690898299217224, 0.43091773986816406, 0.35667282342910767, 0.6703294515609741, -1.9891763925552368, -0.12433550506830215, 0.6557236909866333, -2.3411810398101807, -1.2281545400619507, 1.0758247375488281, -0.23958264291286469, -0.545954704284668, 2.6368956565856934, -0.7272889018058777, -1.3908417224884033, 0.9971950054168701, 0.16325871646404266, 0.7415488958358765, -1.057679295539856, 1.5930755138397217, -0.40775731205940247, 0.3536393642425537, -0.23580263555049896, 0.5481337308883667, 0.19748270511627197, -0.5315186381340027, 0.4146290421485901, 1.8312853574752808, -0.8676590323448181, 0.02865304797887802, 1.1638904809951782, -0.0334702730178833, -0.18077844381332397, 0.5542469024658203, 0.8047366142272949, -1.6184003353118896, -0.14991872012615204, -0.15493017435073853, 1.0697641372680664, -1.758331298828125, 1.2870430946350098, -0.2801913619041443, -0.1898685097694397, -0.7467890977859497, 0.2693692445755005, -0.11613386124372482, 0.9522756338119507, -0.742649257183075, 0.015938352793455124, -1.0108004808425903, -0.5443745255470276, -0.1986585110425949, 0.06860936433076859, -0.38636893033981323, 1.0160768032073975, -1.8788784742355347, -1.6322513818740845, 1.7425717115402222, 1.121022343635559, -0.38312116265296936, 0.7015207409858704, 0.4113003611564636, -0.29761621356010437, -0.9703885912895203, -0.4421992599964142, 1.0999497175216675, -0.14430806040763855, 0.8794078230857849, -0.25190454721450806, -0.6161279678344727, -0.6988216638565063, 1.3617610931396484, 0.3056049942970276, 1.744761347770691, 0.5092299580574036, -0.7157967686653137, -0.10973736643791199, 0.9582842588424683, -0.34349149465560913, -0.8536466956138611, -0.5727159976959229, -0.20852968096733093, 1.0232914686203003, 1.4231234788894653, -0.657197117805481, -1.0294591188430786, -0.9204738140106201, -0.7888309359550476, -0.7733291983604431, -0.12329091876745224, 0.009083897806704044, 0.1401062309741974, -1.0744609832763672, -1.2515554428100586, 1.7644126415252686, -0.020454244688153267, 0.10616526007652283, 0.04231875017285347, 0.32920166850090027, 1.4190480709075928, 0.6199309229850769, -0.35734298825263977, 1.1684458255767822, -0.637116551399231, -0.5027690529823303, -1.9823791980743408, 1.0676970481872559, -0.6362751722335815, 0.0908300131559372, 0.87027508020401, -1.0981777906417847, 0.1207323744893074, -0.7888408899307251, -0.5053671598434448, -0.1710282564163208, -1.7290438413619995, -0.5313480496406555, 0.42290136218070984, 1.801520824432373, 0.049766141921281815, -0.6773278117179871, -1.4449549913406372, -0.49289944767951965, -0.7211211323738098, -1.6140481233596802, 0.09912438690662384, 0.29575541615486145, -0.6069098114967346, 0.2654176354408264, 1.291621208190918, 1.6591265201568604, 0.2985551655292511, -1.236190915107727, 0.7555058002471924, 0.38045597076416016, 0.7492982745170593, -0.635383665561676, 1.3107479810714722, -0.3474685251712799, -1.0331594944000244, 0.5056270956993103, 0.34044766426086426, -0.17889951169490814, -0.6503179669380188, -2.316746234893799, 0.9378418326377869, -0.27852585911750793, -1.9759777784347534, -0.2589583098888397, -1.3925622701644897, 0.7265284061431885, -0.9414036273956299, -1.6911349296569824, 0.23463748395442963, -0.18912425637245178, 0.17318284511566162, 0.762063205242157, 0.04758916795253754, -2.278085708618164, 0.8259880542755127, -1.0654367208480835, 0.3035782277584076, 0.31429699063301086, -1.0351414680480957, -0.4834052622318268, 0.3172670900821686, -1.1636805534362793, -1.2336353063583374, 1.5132451057434082, 0.5882648229598999, 0.003120445879176259, -1.524153470993042, -1.164522647857666, 0.39753487706184387, 0.8808735013008118, -0.11351735144853592, 0.5644670128822327, -0.5779844522476196, -0.45349445939064026, 0.4539045989513397, 0.9530408978462219, -0.7264145612716675, 0.29424989223480225, 0.4968522787094116, 0.11363396048545837, -0.1858670562505722, 0.3918953835964203, -1.3008434772491455, 0.5867713093757629, -0.12505409121513367, 0.24530386924743652, 1.5586285591125488, 0.4063459038734436, 1.5586239099502563, 2.53747296333313, -0.24259056150913239, 1.925445795059204, -0.7491199970245361, 1.2495198249816895, 0.881697952747345, 0.33338332176208496, 0.255614697933197, 0.12831071019172668, -0.21851545572280884, 1.592353105545044, 0.5212874412536621, 0.367124080657959, 0.11119870841503143, -0.18582221865653992, 0.576781153678894, -0.7059677243232727, 0.8600208759307861, 0.9412456750869751, -0.4493711292743683, 1.1830796003341675, 0.586851179599762, 0.41248607635498047, -0.7653787732124329, -0.884404182434082, -0.8765629529953003, 0.4960556924343109, -0.7612279057502747, 0.5271589159965515, -0.8206654191017151, 1.527467131614685, 0.36618882417678833, -0.6846646070480347, 1.8934582471847534, -1.2427366971969604, -0.2760251760482788, -0.6708817481994629, 1.3049700260162354, 1.0235251188278198, 1.6060880422592163, 1.2815014123916626, -0.45260220766067505, 0.7262283563613892, 1.4083054065704346, 0.5051992535591125, -0.23363670706748962, 0.03392217308282852, 0.3999916911125183, 0.773457944393158, 0.7328659892082214, 0.09074563533067703, 2.421433448791504, 0.028097612783312798, -0.9017578363418579, -0.013695862144231796, 0.8998851776123047, -1.4071742296218872, 1.7256344556808472, -0.15735483169555664, -1.4157742261886597, -0.9278171062469482, 0.7386333346366882, -0.26829972863197327, 2.2361302375793457, -1.2548999786376953, -0.8107873797416687, 0.39717593789100647, 0.3540349006652832, -0.5942010283470154, -1.3284354209899902, 1.2850261926651, -0.6989927291870117, 0.1226922944188118, -0.5470386743545532, 0.18235091865062714, -0.2343725562095642, -0.006413048133254051, 1.2042495012283325, 0.4401620924472809, 0.27807971835136414, -1.0991359949111938, 0.022900808602571487, 0.5287555456161499, -0.7320815920829773, -0.7851526141166687, -0.5195068717002869, 1.8666985034942627, -0.9769546389579773, -1.074886441230774, 1.0783988237380981, -0.3781299591064453, 1.5480475425720215, -0.4243268668651581, 0.7750126719474792, -0.33839061856269836, -0.8523827195167542, 1.342970371246338, -0.4724631905555725, 0.5307012796401978, 0.5503565669059753, -1.4749780893325806, -2.502997398376465, 0.7559462189674377, -0.6604655981063843, 1.1191165447235107, -0.5855313539505005, 0.7866794466972351, -1.1397546529769897, -2.043332576751709, 1.4651215076446533, -0.5373444557189941, -1.0466867685317993, -2.9304356575012207, 1.3868434429168701, -0.433895468711853, 0.7383519411087036, -0.16975028812885284, -0.4731508493423462, -0.3455308973789215, 1.1413938999176025, 1.2427362203598022, -0.518419623374939, 0.6200165152549744, -0.1480903923511505, 0.7623233199119568, 0.9271157383918762, -0.21086269617080688, -0.015674905851483345, -0.1494455635547638, 1.3287742137908936, 0.17871995270252228, -0.4753008782863617, -0.11496550589799881, -0.669958770275116, 0.7974095344543457, -0.4399923086166382, 0.8421335220336914, -1.47487211227417, 0.2022087424993515, 0.297285258769989, 0.8818235397338867, 0.6790842413902283, -0.14510171115398407, -0.5946492552757263, -0.3168388605117798, 0.4686671197414398, 0.661658763885498, -1.1760107278823853, -1.3796792030334473, -0.6730178594589233, 0.11297446489334106, -0.9426646828651428, 0.3022810220718384, -0.4261641204357147, -0.30431410670280457, -2.5896637439727783, -1.0996034145355225, 0.703629195690155, 0.25130683183670044, 2.2658016681671143, 0.17512856423854828, 1.1011059284210205, -0.47451409697532654, -0.5327430367469788, 0.7104315757751465, 1.8872509002685547, 1.6991117000579834, 1.0811944007873535, -0.04525226727128029, -1.0422450304031372, -1.0072288513183594, -0.5654340982437134, 1.251582145690918, 0.06382222473621368, -2.388293981552124, 0.028082342818379402, -0.15264347195625305, -0.8510295748710632, -0.7007766366004944, -0.6544947624206543, 0.9296571612358093, 0.4270123541355133, 0.35663604736328125, 1.3568965196609497, 1.8069424629211426, -0.19354146718978882, 0.24875463545322418, -0.35544803738594055, 2.1406314373016357, 0.14284250140190125, 0.783367931842804, -0.3241503834724426, 1.105576992034912, 0.4697034955024719, 1.1061960458755493, -1.1449748277664185, -1.6235623359680176, 0.7374609708786011, 0.47621989250183105, 0.4272252321243286, 0.8972557783126831, 2.3299872875213623, -0.22045458853244781, -0.668167233467102, 0.8259017467498779, 1.9103724956512451, -1.1541012525558472, -0.34070274233818054, -2.7909772396087646, -0.45057356357574463, 0.8053072094917297, -0.07037392258644104, 0.6088361144065857, 0.22597873210906982, -0.3862552344799042, 0.4800961911678314, -0.045399513095617294, -0.7959995269775391, -1.3329166173934937, 0.17873576283454895, 0.5849385857582092, 0.029892627149820328, 0.4174806773662567, 0.54880291223526, 1.9811946153640747, -0.15635770559310913, -1.101248025894165, -0.913824737071991, -1.3923094272613525, 0.8267516493797302, -2.580393075942993, 0.4579838216304779, -1.2170262336730957, 0.7946528196334839, 1.924518346786499, -0.6106874942779541, -0.03118366189301014, -0.6958047747612, -0.7131105065345764, 0.2734020948410034, 1.4311355352401733, 0.8578742146492004, -0.8698328137397766, 1.213255763053894, -0.4408043324947357, -1.9910564422607422, -1.9057035446166992, -0.48071756958961487, 0.8526351451873779, 0.9696279168128967, -0.9371943473815918, -0.5037336349487305, 0.08346472680568695, 0.03608694672584534, -0.20562197268009186, 0.14074456691741943, -1.8320081233978271, -0.3616134822368622, 0.31132611632347107, -0.16413888335227966, 1.7367039918899536, -0.01400720700621605, -0.7750025391578674, 0.25274160504341125, -0.4978142976760864, -0.642839789390564, -0.3449288010597229, 1.1060130596160889, 0.12159740179777145, -1.3358888626098633, -0.29646554589271545, 0.6033095121383667, -0.9350060224533081, 0.9591338038444519, 0.04967949911952019, 0.8576964735984802, 0.6776186227798462, -0.1921122968196869, -0.020711256191134453, -1.6609407663345337, -0.9407361149787903, 0.443658709526062, -0.5671023726463318, 0.2200983464717865, -0.6489423513412476], [1.2266809940338135, 0.8493142127990723, 0.8145638108253479, 0.03221721574664116, 0.867174506187439, -0.33327215909957886, -0.3208751976490021, -0.17544473707675934, -0.5102939605712891, -0.4252074062824249, -0.4717367887496948, -0.8353825807571411, -0.8942036032676697, -1.1623802185058594, -1.1797555685043335, -0.2866560220718384, -0.31737926602363586, -2.714949607849121, 0.6888798475265503, -1.981285810470581, 1.0684255361557007, -0.2608337700366974, -1.2484346628189087, -2.2641425132751465, 1.6879974603652954, -0.6626958847045898, -2.2775349617004395, 1.5376099348068237, 0.05597381293773651, -0.6558640599250793, -1.2029109001159668, -0.11366377770900726, -1.6893062591552734, 1.2550764083862305, -0.36082854866981506, 1.4120070934295654, 1.6388307809829712, -1.8304400444030762, -0.23250886797904968, 1.4992218017578125, -1.505030870437622, -0.3045922815799713, -0.3068847358226776, 1.8615241050720215, -0.7688041925430298, -0.4378075897693634, 0.4732193946838379, 0.17560353875160217, -0.33876487612724304, 0.5713239908218384, -0.8337656259536743, 0.8713828325271606, -0.02261713147163391, 1.5813376903533936, 0.9256439208984375, -0.14134956896305084, 0.09733182936906815, -1.465920329093933, 0.2975948750972748, -0.06689579039812088, -1.110267162322998, -0.5606481432914734, -0.15673133730888367, 0.324773907661438, 0.8664416670799255, 2.2707512378692627, 0.446887731552124, 0.032410718500614166, -2.191340446472168, -0.006718937307596207, 0.18370547890663147, 2.508070707321167, 0.5719200372695923, -0.7744518518447876, 0.7705562710762024, -0.03369169682264328, 0.4330460727214813, -0.12544964253902435, -0.46497002243995667, 0.9350046515464783, -0.925852358341217, 0.7399489283561707, -0.0643698051571846, 0.3448793888092041, -0.3227951228618622, 0.11792974174022675, 0.6523460149765015, -0.14777182042598724, 0.5570599436759949, 1.987623691558838, -1.0224545001983643, 0.6118077039718628, -0.5250954627990723, 0.015543130226433277, 0.5531646013259888, -1.0311721563339233, -0.5145992636680603, -0.7464489340782166, -0.38235339522361755, -1.2599759101867676, 0.7525184750556946, 0.10036522150039673, 1.1494777202606201, 3.578653335571289, -0.6592726111412048, 0.038312699645757675, 0.23582032322883606, 0.3975260853767395, 0.3955817222595215, 1.153146505355835, -0.11721301823854446, 0.7023473381996155, -1.3642810583114624, 0.7116044759750366, -2.313481569290161, 0.5613868832588196, 0.15022392570972443, 1.9315563440322876, 0.41224363446235657, 0.04385320842266083, -0.8044903874397278, 0.02920067124068737, -0.17217393219470978, -0.766287624835968, 0.27224093675613403, 0.952057957649231, 0.9115425944328308, 0.5021712779998779, -1.270778775215149, 0.6869094371795654, 0.9907407164573669, -0.6017913818359375, 0.07067360728979111, -0.3760705292224884, 0.010696491226553917, -0.20995840430259705, 0.24555443227291107, -0.09353267401456833, 1.4669688940048218, 0.21326373517513275, -1.2572201490402222, 0.9910666942596436, 2.524693727493286, 0.9162033796310425, -1.175792932510376, -0.2750311493873596, 0.19979703426361084, 0.07524321973323822, 0.6634238362312317, 0.4365856349468231, -1.5246607065200806, -0.6809439659118652, 0.512973427772522, 0.9523200392723083, -0.3948615789413452, 0.8633130788803101, 2.0577149391174316, -0.8688340187072754, -0.2881399989128113, -0.8989046812057495, 1.9063358306884766, 0.1164848580956459, 0.8707005977630615, -1.3854060173034668, 0.40935179591178894, 1.044543981552124, -0.233831524848938, 0.8042136430740356, -0.22594304382801056, 2.732337474822998, -0.32439932227134705, -0.6932784914970398, 0.30844488739967346, -0.5380431413650513, 0.41945141553878784, -1.8702021837234497, -0.8416098952293396, -0.27144861221313477, -0.6922100186347961, -0.10031381249427795, 1.2375173568725586, -0.5235913395881653, 0.5438085794448853, -0.4035370647907257, -0.42652106285095215, -1.7381831407546997, 1.1782987117767334, -0.8967435359954834, 0.1240389496088028, 0.38497254252433777, 0.18973132967948914, 0.43297648429870605, -0.517206072807312, -0.920931875705719, 0.7277272939682007, 1.3466721773147583, 1.4995996952056885, 0.10435870289802551, 0.9744997024536133, 1.668310523033142, 1.63006591796875, -1.2969796657562256, -0.15006941556930542, -0.08860865980386734, -1.7693936824798584, -0.08492070436477661, 0.5155500769615173, 3.132964611053467, 1.4607208967208862, -0.5510353446006775, -0.5175397992134094, 1.5418719053268433, -0.7134503126144409, 1.3372869491577148, -2.597482919692993, -0.9948534369468689, -1.4149614572525024, 0.14747047424316406, 1.044167160987854, -0.8029226064682007, 1.110975980758667, 0.5818007588386536, -2.212739944458008, 0.5000842213630676, -0.1852773278951645, 0.381046324968338, 1.3665317296981812, 0.34310945868492126, 0.5103018879890442, 0.34823715686798096, -0.7855405807495117, 0.8451123833656311, -0.855423092842102, 1.3403679132461548, -2.444903612136841, -0.31935039162635803, -0.5125814080238342, 0.3207726776599884, -0.05459870025515556, -0.5550360679626465, -0.47891342639923096, 0.15375512838363647, -0.7186905145645142, 0.4171908497810364, -1.2819808721542358, 0.2962067425251007, -1.5963274240493774, -1.9629161357879639, 1.3732942342758179, 2.0998635292053223, -0.04934818297624588, -0.005201918072998524, -1.7147026062011719, -0.5287149548530579, 0.6141794323921204, -0.3549772799015045, -0.044602278620004654, -0.6610419750213623, -0.17795220017433167, -0.06889836490154266, -1.7017048597335815, -1.906497836112976, -1.1876674890518188, -0.9659767746925354, 0.25147759914398193, -0.11399827152490616, -1.4308598041534424, -0.8586316108703613, -0.9363471865653992, 1.519850730895996, 1.4546748399734497, -1.7332956790924072, 0.40911585092544556, -1.1475850343704224, -1.2345750331878662, -1.3283717632293701, -0.13413013517856598, -1.2243396043777466, 1.0897034406661987, -1.8622758388519287, -1.1290957927703857, 1.1103540658950806, 0.8947167992591858, 0.8878636956214905, -0.7189991474151611, 0.7097323536872864, -0.5232797265052795, 0.19272252917289734, -0.4718629717826843, 0.5160965323448181, -1.4793636798858643, 0.28031811118125916, 0.597263753414154, 0.19868798553943634, 1.6038758754730225, -0.2568308711051941, 0.0398935042321682, -0.9841158986091614, 0.8427606821060181, -0.08422281593084335, 1.4368659257888794, 0.630619466304779, -0.6297720670700073, -0.075265072286129, 1.0041446685791016, 1.4258943796157837, 0.39304232597351074, 0.0543755479156971, 1.0022495985031128, -2.0778732299804688, -1.3282235860824585, -0.020902303978800774, 0.8424187898635864, 1.5293796062469482, 0.6243253946304321, 0.9413079619407654, 0.08859533816576004, -0.445663183927536, 2.2363553047180176, -0.02084190957248211, 0.03134509176015854, 0.1454545557498932, -1.2303674221038818, -0.034445472061634064, -0.5707111358642578, -0.6119369268417358, -1.4659854173660278, -1.3616175651550293, -0.733893632888794, 1.4911998510360718, -0.020572450011968613, 0.06086338683962822, -0.8478456735610962, 0.18864810466766357, 0.40491488575935364, -1.4259276390075684, 0.574165940284729, 0.22971177101135254, -0.45569100975990295, -1.3204562664031982, -0.5948827266693115, 0.826122522354126, -1.0163607597351074, 1.4839760065078735, 1.0155178308486938, -0.6837418675422668, 1.4134657382965088, 0.20116856694221497, 0.6552416682243347, 1.1083840131759644, 1.6021287441253662, 0.0017809770070016384, 0.48351967334747314, -0.24070803821086884, 1.338452696800232, 0.8786020278930664, 1.1885693073272705, -0.13324497640132904, 0.7316242456436157, -0.7147666811943054, 0.19182847440242767, -0.6539705991744995, 0.9437961578369141, -1.5824264287948608, -1.067089319229126, -0.43950363993644714, 0.07584094256162643, -0.7723796367645264, -0.7755129933357239, -0.08041930198669434, 0.39204418659210205, -0.14402556419372559, 0.11406832933425903, 0.7309034466743469, -0.2884414792060852, -0.9346951842308044, 1.0242127180099487, -0.7376202940940857, -1.9899243116378784, -0.5013065934181213, -1.7478502988815308, 0.4033384919166565, -0.6119197607040405, -0.34035587310791016, -0.35760608315467834, -0.9149953126907349, 0.403033584356308, 1.0892975330352783, -0.22373153269290924, -0.054928261786699295, -0.3256424069404602, -0.003198344260454178, -1.8404017686843872, 0.7595441937446594, -0.44768545031547546, -1.0235934257507324, 1.134216070175171, 0.13645583391189575, -0.4596192538738251, 1.1504185199737549, 1.1051325798034668, 0.2453843057155609, 0.9332413673400879, -1.9830174446105957, 0.41213321685791016, 1.0983296632766724, -1.1570080518722534, -0.3220044672489166, 0.20844687521457672, -0.3930058479309082, 0.826626718044281, 0.1853296309709549, -0.2163383960723877, -1.0447436571121216, -0.37256920337677, 0.22960075736045837, -0.24868468940258026, 0.723371684551239, -0.10752511769533157, -0.7050425410270691, 2.207594156265259, -0.20995624363422394, 0.04390605911612511, -0.19546210765838623, -0.7976505756378174, -0.8562206625938416, 0.43016183376312256, -0.5921847820281982, -0.7994043827056885, 1.028550148010254, -1.0225794315338135, 0.611965537071228, -0.2745135724544525, -1.3911340236663818, -1.3668591976165771, -1.007206678390503, -0.6139347553253174, 0.9380148649215698, 0.19221119582653046, -0.8907855153083801, 0.27416881918907166, -0.16461704671382904, 1.5795015096664429, 0.0730673298239708, 0.4765414595603943, 0.2809387743473053, -0.5497750043869019, -0.2067108154296875, -0.564708948135376, -0.38916605710983276, 0.8709532022476196, -0.3534350097179413, 0.6287211179733276, 0.07513286173343658, 1.4306915998458862, 0.5994120240211487, -2.091909170150757, -0.1497928649187088, 0.4228874146938324, -1.220418930053711, -0.19204765558242798, 0.3610507845878601, 0.19756056368350983, -0.5045921206474304, 0.11553864926099777, -0.06506610661745071, 0.24910540878772736, -0.47945016622543335, 0.21445825695991516, -0.5146290063858032, 1.1478252410888672, 1.6415272951126099, 0.5143053531646729, -1.761681079864502, -0.20094844698905945, 1.2667970657348633, 0.18272092938423157, 0.8030716180801392, 0.0915973037481308, -1.347984790802002, -1.2030465602874756, -1.1267945766448975, -0.6565590500831604, -0.7156470417976379, 0.7605397701263428, 0.32401564717292786, -1.0718027353286743, 0.8213520050048828, 0.7987818121910095, 0.47785794734954834, -1.6556881666183472, -0.5839372873306274, 0.9891008138656616, 0.32869961857795715, 1.5125277042388916, -0.7897509336471558, 0.8775302171707153, 1.6163095235824585, -1.047065019607544, 0.41856011748313904, 1.365224838256836, 0.11146527528762817, 0.3872647285461426, -1.158551812171936, -0.2961806356906891, 0.4241587519645691, 0.4033913314342499, -0.592389702796936, -0.3322451710700989, 1.9129022359848022, 0.6263586282730103, -1.1506351232528687, -0.3930714428424835, 0.42336830496788025, 0.6864105463027954, -1.5701160430908203, -1.688713550567627, 0.988217294216156, -2.2028260231018066, 1.149375081062317, 0.2923179864883423, 1.3581938743591309, -0.24212224781513214, -0.21632348001003265, -1.0771723985671997, 0.7613687515258789, -0.05141261965036392, -1.0172083377838135, -1.0817523002624512, 0.003774563316255808, 0.8910187482833862, -0.675793468952179, 0.7700002193450928, 0.005685539450496435, -0.6916659474372864, 0.2807181477546692, -0.217664897441864, 2.6790926456451416, -0.5953054428100586, 0.7004629969596863, 1.4157534837722778, 0.07709746062755585, -0.10205898433923721, 2.0395452976226807, 3.179495334625244, -0.6728206276893616, 2.5848095417022705, -0.5609414577484131, -0.14060458540916443, -0.3691454529762268, 0.4608256220817566, 1.3540879487991333, 1.2081483602523804, -1.5235618352890015, -2.0914056301116943, -1.2716726064682007, -0.737694263458252, 0.7515230774879456, 0.019681887701153755, -2.6399147510528564, -1.0195637941360474, -0.379834920167923, 0.9546012878417969, 0.010513207875192165, 1.0051523447036743, -1.4301444292068481, 0.7415766716003418, 0.3894336223602295, 0.1484745591878891, -0.3496209681034088, 0.8867208361625671, -1.558701992034912, -1.9726552963256836, 0.08452432602643967, 0.7857362627983093, -1.0110355615615845, 0.6227632761001587, 1.593320608139038, -0.8879789710044861, -1.2164924144744873, -0.061545856297016144, -0.5381471514701843, -1.3213706016540527, 0.8040820956230164, 1.240858554840088, 0.5431370139122009, -0.06061607599258423, -1.1681650876998901, 0.3716109097003937, 1.1771764755249023, -0.0651259571313858, -0.13358472287654877, 0.8472719788551331, -0.0228891484439373, 1.8383387327194214, 0.7070123553276062, -1.2211347818374634, -1.0675395727157593, 1.766671895980835, -0.20167987048625946, 0.018197644501924515, 0.2465231865644455, -0.0910605937242508, -1.767214059829712, -1.195507287979126, -0.4826558232307434, -0.9205715656280518, -0.2354266494512558, 0.061361175030469894, 2.1075947284698486, 0.6256568431854248, 1.8871960639953613, -0.6383051872253418, 0.1750766485929489, -0.6006821393966675, -2.5652456283569336, 0.07659953087568283, -0.6843445897102356, -1.320627212524414, -0.5987318754196167, 1.4685852527618408, -1.7992135286331177, 0.34727588295936584, -0.2916077673435211, -0.5528845191001892, -1.1758465766906738, 0.33626440167427063, 0.014988509938120842, 0.444136381149292, -0.37803882360458374, 0.5209933519363403, 0.15476006269454956, -1.1290301084518433, -0.5530354380607605, 0.41731634736061096, -0.22489076852798462, 1.7193355560302734, 0.29042184352874756, 1.3914248943328857, 1.6073071956634521, -1.8635787963867188, -0.8579864501953125, 0.7843931317329407, -0.7268401384353638, 0.4482063353061676, 1.0477674007415771, 0.09905124455690384, -0.11065509170293808, 0.7782869338989258, -0.26518309116363525, 1.0401787757873535, -0.8406247496604919, -0.4816536605358124, -0.035685040056705475, -0.8247873187065125, -0.17539744079113007, -2.5847208499908447, -0.6871708631515503, -0.7831290364265442, -1.3644260168075562, -0.5587779879570007, -1.0032272338867188, -0.12857337296009064, -1.8407080173492432, -0.16573530435562134, -0.9576725959777832, 0.522225022315979, 0.7262234091758728, 0.5889240503311157, 0.8750052452087402, 0.9191584587097168, -1.3124439716339111, -1.6217893362045288, -0.7703441977500916, 0.6545163989067078, -0.3510344326496124, 0.6960037350654602, -0.16448938846588135, 0.3900395631790161, 0.5652709603309631, 0.5103069543838501, -1.0888137817382812, -1.3779081106185913, -1.0800575017929077, 0.3049779534339905, 1.039197325706482, -0.36345988512039185, 0.8214720487594604, -0.2510708272457123, 1.483693242073059, -1.7036091089248657, -0.6029684543609619, -0.41081592440605164, -0.8978142142295837, 0.27314549684524536, 1.0926909446716309, -1.3000686168670654, 0.19930829107761383, -0.6998234391212463, -1.3327921628952026, 0.9722987413406372, 0.6112284660339355, 0.22793227434158325, -1.6414637565612793, -2.4069368839263916, -0.13406537473201752, 0.14835648238658905, -1.0445232391357422, 2.0349695682525635, -0.24451439082622528, -0.47318005561828613, 2.2760093212127686, 0.401837021112442, -1.3437262773513794, 0.1655421257019043, 0.6848342418670654, -2.551068067550659, 0.9165855646133423, 1.0693775415420532, 0.8556154370307922, -0.6670394539833069, 0.09745366126298904, 0.27528566122055054, 2.440300941467285, -0.793379008769989, -0.42495065927505493, -0.888558566570282, -1.173036813735962, 0.9232040047645569, 1.236849069595337, 0.8272470831871033, 1.4880433082580566, -0.3529459238052368, 0.38275542855262756, -0.9803576469421387, 0.18067243695259094, -0.47748321294784546, 1.4013104438781738, 0.11452367901802063, 0.015799080953001976, -0.37672340869903564, -1.5943619012832642, -1.7726459503173828, 0.6145875453948975, -1.4498744010925293, -0.8845602869987488, -0.4181811213493347, 0.22233977913856506, -0.4694181978702545, -0.5073145627975464, -0.4524162709712982, 0.22594544291496277, -1.3640854358673096, 0.04795059934258461, 0.26928165555000305, -0.23484818637371063, 0.4906502068042755, 1.5649031400680542, -0.3657337725162506, 0.027171140536665916, -0.4483153522014618, 1.9109402894973755, -0.1582089364528656, 0.25659722089767456, -1.2805814743041992, -0.7007072567939758, 0.8514704704284668, -0.3267379403114319, -0.3989473581314087, -1.3785905838012695, -0.2576104402542114, 0.8258947134017944, 0.19750896096229553, -2.132429599761963, -1.0451236963272095, -0.17273522913455963, 1.6178971529006958, -1.61601984500885, -1.5080175399780273, -0.21598148345947266, 0.3009069859981537, -0.9021329283714294, -0.8773592114448547, -1.3217946290969849, -1.6049013137817383, 1.054317831993103, 0.8456921577453613, -0.5609197616577148, 2.7205967903137207, 1.2853902578353882, 0.661786675453186, -1.4114959239959717, -0.817979097366333, 1.6261446475982666, -2.607388496398926, -1.162294626235962, -1.2878656387329102, -1.612918496131897, 0.3446658253669739, -1.558457374572754, -0.030263016000390053, 0.8656017780303955, 0.38627955317497253, 1.66202712059021, 1.1343241930007935, -0.23391658067703247, -1.2314302921295166, 0.4968748986721039, -1.8995765447616577, -0.26234936714172363, 1.3071823120117188, 0.16897179186344147, -1.1541134119033813, 0.5300229787826538, 1.358309268951416, -1.1056511402130127, -1.7188541889190674, 0.0004978784709237516, 0.9722710847854614, 1.0093281269073486, -1.0196903944015503, -0.3909567594528198, -0.2553205192089081, 0.04288424924015999, 2.0297691822052, -0.06468497216701508, 0.6150711178779602, -1.0012863874435425, 0.8145521283149719, 1.518356442451477, 0.8148291707038879, 0.8771981596946716, 1.199565052986145, -0.6852668523788452, 0.27853924036026, 1.3145750761032104, -0.2784758508205414, 1.154167890548706, 0.6206352710723877, -1.6202691793441772, 0.7483385801315308, 0.5096530914306641, 0.7100204825401306, -0.42274096608161926, 0.6266215443611145, 1.8505443334579468, 0.45751604437828064, 1.4926429986953735, 0.7799124717712402, -0.8536617755889893, -0.42809078097343445, -1.319082498550415, 0.30204474925994873, -0.30356135964393616, -0.6449942588806152, 0.8326539397239685, -0.6867220997810364, 0.6569170951843262, 0.6004505157470703, -1.329160451889038, -0.5948602557182312, 0.1463063806295395, -1.6889060735702515, 0.17495448887348175, 0.34558793902397156, -0.6815087795257568, 0.2862022817134857, -0.10204896330833435, -0.11447012424468994, -1.739652395248413, 0.34658557176589966, -0.48752591013908386, 0.21726949512958527, 0.45755669474601746, 1.4613505601882935, 0.283719927072525, -1.7360872030258179, 0.6090472936630249, -0.03553593158721924, 0.620763897895813, -1.331630825996399, 1.278526782989502, 1.8683098554611206, 0.06071209907531738, -1.9237598180770874, 0.6186047196388245, 0.5423616170883179, -1.734094500541687, 1.4987589120864868, -0.794585645198822, -0.47720634937286377, -0.35347139835357666, -0.41546207666397095, -0.1624222993850708, -0.049763273447752, 0.8243937492370605, 0.22178387641906738, 0.06306004524230957, 0.07404811680316925, 1.157342791557312, -1.1876534223556519, 1.6899276971817017, 1.3452051877975464, 1.1133605241775513, 2.156993865966797, -0.9678148031234741, 0.34135982394218445, -0.8718730807304382, 0.29399043321609497, 0.7424803376197815, -0.45444217324256897, 0.41199830174446106, -1.0032713413238525, 0.4102376699447632, 1.0009844303131104, -0.005442847032099962, 1.6885930299758911, 0.4844738841056824, 0.28294873237609863, -1.0511064529418945, 0.22339728474617004, 1.4490081071853638, 0.6071528196334839, -0.09772193431854248, 1.093668818473816, 1.3782769441604614, 0.1871522217988968, 2.022395133972168, 0.009895486757159233, 1.861840009689331, -0.9767771363258362, 1.7637139558792114, -0.9668765664100647, -0.44697311520576477, -0.003733213059604168, -1.8583482503890991, -1.1631407737731934, 0.3563512861728668, 0.36556580662727356, -0.975701093673706, 0.29316961765289307, -1.4249041080474854, -2.5273070335388184, 0.022276824340224266, -1.6675827503204346, -1.066577434539795, 0.8433144688606262, 2.396353244781494, 0.6170953512191772, 0.09371618181467056, 0.6910340785980225, 1.7628191709518433, 1.559299349784851, 0.45491981506347656, -1.4822454452514648, 0.31439337134361267, -0.25197744369506836, -0.8867030739784241, -0.38823506236076355, 0.12367546558380127, -0.5228977203369141, 0.5294618606567383, 0.38168054819107056, -0.9461709260940552, -0.10404899716377258, -0.7742053866386414, -0.7874168753623962, 0.7292328476905823, -0.4623660743236542, -0.3958565592765808, 0.8795158267021179, -1.4403011798858643, -2.083672523498535, 0.4707338511943817, 0.8615714907646179, 0.541838526725769, -0.9079842567443848, 1.2762787342071533, -2.4243242740631104, 1.8598865270614624, -0.23290157318115234, -1.6052231788635254, 0.7911699414253235, 0.78010094165802, -0.16801616549491882, -0.6515582799911499, 0.08675224334001541, 1.4347138404846191, 0.41723018884658813, -0.3411298990249634, 0.29956597089767456, -0.5482965707778931, -0.9502376914024353, 1.0174702405929565, 0.9302162528038025, 2.053797483444214, -0.9944432377815247, 0.9164335131645203, -0.9088230729103088, -0.24765050411224365, 0.6357970237731934, 0.19234226644039154, -2.1678740978240967, 1.0385276079177856, -0.6641950607299805, -0.41423508524894714, -0.9013994336128235, 0.4180375039577484, 0.1626584380865097, -0.7259383201599121, 1.5323110818862915, 0.3836708664894104, 0.5489532947540283, -0.09346972405910492, 2.896707534790039, -0.6895575523376465, -1.6685937643051147, 0.45352035760879517, -1.0045406818389893, -1.0900840759277344, -0.4804123342037201, 0.2687940299510956, 0.7116320133209229, 0.48266199231147766], [-0.1408800631761551, 0.35062375664711, 1.8523815870285034, -1.1539835929870605, -0.940664529800415, -1.9052995443344116, -0.5204989910125732, -1.0483756065368652, -1.2769001722335815, -1.1821675300598145, 1.0071250200271606, -1.2465022802352905, 0.10774238407611847, 0.32525891065597534, -1.7846863269805908, 0.009138960391283035, -0.9363534450531006, 0.7404187917709351, 0.27560681104660034, 0.16782620549201965, -0.8382423520088196, -0.8488515019416809, 0.18300071358680725, 0.9741507768630981, -1.0173550844192505, -0.021220533177256584, 0.07388743758201599, 1.8381388187408447, 0.2300858497619629, 2.344125747680664, 0.43863144516944885, -1.0856794118881226, 0.021269863471388817, 0.5817674994468689, 1.2410385608673096, -1.4736485481262207, -0.2284403294324875, 0.012644359841942787, 0.6806001663208008, -0.3796961009502411, -2.228898048400879, -0.7237066626548767, -1.5240429639816284, 0.8116238713264465, 0.36955517530441284, 1.5059338808059692, 0.31147393584251404, -0.46521061658859253, 0.9429660439491272, -0.24544978141784668, 0.06545455753803253, -2.132155418395996, 0.3108311593532562, -1.5745991468429565, 0.23647388815879822, -0.058823127299547195, 0.15047189593315125, -1.0843629837036133, -0.4265766441822052, 0.43708592653274536, 1.2175360918045044, 0.267403244972229, 1.0814276933670044, 1.572754144668579, -1.482454776763916, -0.413430392742157, -0.40364590287208557, 1.4291623830795288, -1.1295446157455444, -0.408395916223526, 0.08550557494163513, 0.3908841907978058, 0.4383256435394287, -0.7452076077461243, 0.7723191976547241, -0.17735381424427032, 1.3302541971206665, -0.7738750576972961, -0.1848476082086563, -1.1249722242355347, 1.0470317602157593, -0.9014831781387329, -1.9484367370605469, 0.43344321846961975, -0.26710066199302673, -0.8772038817405701, -1.2327176332473755, -0.6971826553344727, -0.55523282289505, -0.16978693008422852, -0.7163165807723999, -0.45551809668540955, -2.3746070861816406, 1.6670085191726685, -0.36026355624198914, 0.13918575644493103, -1.4350098371505737, 1.2037323713302612, -0.5874372720718384, 0.4342510998249054, 0.4328015148639679, 0.02629498764872551, -0.17274320125579834, -2.461561679840088, 0.6343832612037659, 0.718231737613678, -0.34955033659935, -0.9899017214775085, -1.0525027513504028, 0.2069302201271057, -0.6606242060661316, -0.37424585223197937, -1.1293673515319824, -0.7117519378662109, -0.5424665808677673, 0.7699505090713501, -0.05868548899888992, 2.1771445274353027, 0.76409912109375, -0.1603081077337265, -1.915543794631958, -0.09786660969257355, 0.3392510712146759, -0.07203636318445206, -0.9180800318717957, 1.5506733655929565, -1.3269048929214478, 0.41431841254234314, 0.7545855045318604, 1.198723554611206, -0.1927698254585266, -0.6280599236488342, 0.8496082425117493, 0.6759665012359619, -0.5500569343566895, -0.7136497497558594, -0.3222540616989136, -1.507004976272583, -0.27923890948295593, 0.27780881524086, -0.43415242433547974, 0.06703431904315948, 1.003024935722351, 1.5893313884735107, -0.38215872645378113, -0.5424237847328186, -0.9181923270225525, 0.29371345043182373, -0.6415606737136841, -2.2400293350219727, 0.6052707433700562, -0.057140327990055084, 1.6846250295639038, -0.1838281899690628, 1.2518305778503418, 0.27342092990875244, -0.4305938184261322, -2.085153579711914, -1.3387575149536133, -1.565087080001831, -0.42609506845474243, -0.63823401927948, -0.3660523295402527, 1.8197988271713257, -1.062821388244629, 0.5589599013328552, 0.13597863912582397, -0.1358683854341507, -1.2741000652313232, 0.18748590350151062, -0.3684084117412567, 1.4534361362457275, -0.5946880578994751, -1.4447039365768433, -1.3197342157363892, 0.2684512734413147, 1.027846097946167, 0.08830946683883667, 0.4373891353607178, -1.532791256904602, 0.17375548183918, -1.5208739042282104, -0.18815450370311737, -0.5291081666946411, 0.05601958930492401, 1.2563735246658325, 0.4205615520477295, 0.9044956564903259, -0.7962473034858704, -0.13079553842544556, 1.3763573169708252, 1.3606624603271484, -0.34406352043151855, -0.3685779571533203, 1.1516607999801636, 0.33427131175994873, -0.6909245848655701, -0.09158002585172653, 0.37479427456855774, -0.2704976499080658, -1.2412633895874023, 0.7823009490966797, 0.021372616291046143, 1.3211849927902222, -1.19577956199646, 1.5017001628875732, -0.9385253190994263, -0.26277148723602295, 0.28953197598457336, -1.1951978206634521, -0.34345924854278564, -0.7701910138130188, 0.08265410363674164, -0.9902495741844177, 0.36856648325920105, 0.254716157913208, 0.24444396793842316, 0.9641357064247131, -1.0612324476242065, 0.27930784225463867, 1.970636010169983, -0.28815576434135437, -0.1897038072347641, 1.1718075275421143, -1.4740076065063477, 0.9768127202987671, 0.12611550092697144, -1.0022281408309937, 0.5767057538032532, 1.101188063621521, -0.20841830968856812, -0.04103884473443031, -0.8407555818557739, -0.08293955028057098, 0.7084778547286987, 0.6483394503593445, -0.2904367446899414, -0.7609130144119263, 1.7291905879974365, -0.27702003717422485, -0.803500235080719, -1.6952265501022339, -2.608935594558716, 0.16224552690982819, 0.14687196910381317, -0.20391739904880524, 0.07499729841947556, -0.5381002426147461, 1.2754892110824585, 1.4637447595596313, -0.7256821393966675, -1.0368956327438354, 1.4721736907958984, 0.5673410892486572, 1.8791723251342773, -0.12850652635097504, 0.004989829380065203, 0.3291427195072174, -1.5097112655639648, 0.27594026923179626, 0.11427256464958191, 1.751173734664917, -0.14963938295841217, -0.37503495812416077, 0.4660537540912628, -0.5518612861633301, -0.7643817663192749, -0.04744300991296768, 1.5962119102478027, 0.34321972727775574, -0.7082024812698364, 0.4793008863925934, 0.5805217027664185, -0.8538413643836975, -0.6580798029899597, -0.268475204706192, 0.40775641798973083, -1.3175443410873413, 0.299650639295578, -0.5047986507415771, -1.0533696413040161, 1.2552748918533325, -1.6948214769363403, 1.3989495038986206, 0.15673711895942688, 1.5265144109725952, -0.24250152707099915, -0.433771550655365, -2.1053144931793213, -1.9195678234100342, 2.075183153152466, 1.2791180610656738, -1.1355611085891724, -0.12130270898342133, 0.29445135593414307, 0.04186178743839264, -0.0930439904332161, -0.6974839568138123, -0.3620644509792328, 0.541560709476471, 0.4259870648384094, -1.9940354824066162, -1.15517258644104, -0.5851842164993286, -1.508284091949463, 0.6870543360710144, 0.31544312834739685, -1.0719786882400513, -0.4539344310760498, -1.7817881107330322, -0.15834245085716248, -1.1668702363967896, 2.292473077774048, 2.2545206546783447, 0.3021775484085083, -0.4800475239753723, 0.4945608973503113, 1.210194706916809, 0.21201635897159576, -1.2450002431869507, -0.9537058472633362, 0.8151955604553223, 0.6319919228553772, -0.09210137277841568, -1.0635805130004883, 0.29190152883529663, 0.2179350107908249, 0.017181523144245148, -0.18238675594329834, 1.3664844036102295, 0.5155221819877625, -0.16798251867294312, -1.2036402225494385, -0.7526648044586182, -0.026862533763051033, -0.2398141324520111, 2.0836315155029297, 1.0639325380325317, 1.5143383741378784, -1.1914530992507935, 1.9927144050598145, 0.5512430667877197, 1.6044132709503174, -0.8960243463516235, -0.11250708997249603, 0.11093027144670486, 0.04358820244669914, -0.561797559261322, 0.288782000541687, 0.8020802736282349, 1.252089500427246, -1.6958304643630981, -1.2258316278457642, 0.4003480076789856, -0.7672900557518005, -0.8933658599853516, -0.8524554371833801, 0.364288330078125, -0.7065995335578918, 0.23979315161705017, 0.0743822380900383, -0.7588991522789001, 0.7877001166343689, 0.2659313976764679, -0.0858084037899971, 1.499375343322754, 0.35268059372901917, -0.406173437833786, 0.8580484390258789, 0.36637285351753235, -1.3477416038513184, 0.31680944561958313, 0.9067346453666687, 0.34459739923477173, -0.33038637042045593, -0.30702534317970276, -0.32025256752967834, -0.6933534741401672, -2.4272818565368652, -1.9668490886688232, 0.49091702699661255, -1.4267195463180542, -1.7050458192825317, 0.4664420187473297, 0.2265578806400299, -0.08053980767726898, -1.7477471828460693, -0.10558747500181198, -1.2180078029632568, -0.41758015751838684, 1.8270938396453857, 0.6380248069763184, 1.2836127281188965, -0.7822225093841553, -0.20348504185676575, 0.3663634955883026, 0.6575082540512085, 0.5183216333389282, -1.4926362037658691, 0.4963171184062958, -0.243096724152565, 0.607826828956604, 0.9866319298744202, -1.215238094329834, -0.5197291970252991, -0.6582289934158325, -1.3213492631912231, 0.7458742260932922, 0.6600946187973022, -0.35644277930259705, -0.7520208358764648, 0.3679661750793457, 0.36195850372314453, 0.5667148232460022, 0.21590207517147064, -0.6945904493331909, -0.1085355281829834, -0.287259042263031, 1.2007358074188232, -0.6888760328292847, 1.849675178527832, -0.13787098228931427, -0.055015236139297485, -1.8052968978881836, -1.4061557054519653, -1.1507350206375122, -0.2574702501296997, 0.33803749084472656, 0.029862675815820694, 2.6949539184570312, 1.8964942693710327, 0.7651688456535339, 0.2222352921962738, 0.3377569615840912, 0.09226839989423752, 0.2653670608997345, 0.10784340649843216, -1.4211034774780273, 2.1465625762939453, 1.0790796279907227, -2.1743123531341553, 0.6895845532417297, -0.3535817861557007, 0.2574480175971985, -2.4585812091827393, -1.1801221370697021, 0.17560230195522308, 1.766079306602478, -0.7496251463890076, 1.5322914123535156, -2.8536031246185303, 0.3177412152290344, 1.6857950687408447, 1.5537638664245605, -0.47062698006629944, -0.6232571005821228, 0.8324399590492249, -1.7794541120529175, -0.6557765603065491, -0.43876832723617554, 1.3298324346542358, 1.812947154045105, -0.07634525001049042, 1.3606767654418945, -1.228662133216858, -0.6747384071350098, -0.9379573464393616, -0.8004217147827148, 0.3440695106983185, 2.889816999435425, 0.01949586719274521, 0.04531837999820709, -0.7343349456787109, -1.2157533168792725, 0.37542611360549927, -1.521317481994629, -0.14881667494773865, 1.624704360961914, 0.63434898853302, -0.16562432050704956, -0.5821675658226013, 0.6856276392936707, -0.31673216819763184, 0.39080777764320374, 1.1557337045669556, 0.5545944571495056, 0.505599319934845, 0.6976747512817383, 1.3549537658691406, -1.141134262084961, 0.2664774954319, 0.27392247319221497, 0.8773362636566162, 0.02902981825172901, 0.6710996031761169, -1.88996160030365, 1.5515260696411133, -1.7697129249572754, 0.3278692066669464, -0.6292106509208679, 1.0684218406677246, 0.5041750073432922, -0.48104795813560486, -1.0654079914093018, 0.25177550315856934, 0.5611386299133301, 0.995478630065918, -1.2057355642318726, -0.23707784712314606, -0.5269187688827515, 0.521958589553833, 1.9793660640716553, -2.3355612754821777, -0.9816572666168213, -1.260486125946045, 0.06384160369634628, 1.0953019857406616, 0.005381942726671696, 1.3455636501312256, 2.512117624282837, -1.2374650239944458, 0.6613460183143616, 0.8500733375549316, 0.24679222702980042, -0.07335592806339264, 0.007207488175481558, 1.3556197881698608, -0.12423518300056458, -0.16709324717521667, 0.013691937550902367, -0.7474178075790405, 0.07638860493898392, -0.49076077342033386, 0.9731937050819397, -2.345329523086548, 1.1787633895874023, 0.039197638630867004, -2.062347650527954, 0.3156406879425049, -1.1878477334976196, 0.10402209311723709, 0.005050174426287413, 2.0294435024261475, -0.13735786080360413, 0.13667111098766327, 0.8976658582687378, -1.8689322471618652, -0.9472400546073914, -0.35210517048835754, 1.16177499294281, 0.16226589679718018, 0.6045576930046082, 1.6905361413955688, -2.0211002826690674, -0.9727867245674133, -2.0922884941101074, 0.24404557049274445, -0.6245574951171875, -0.9283689856529236, 0.1766258329153061, 0.08538810908794403, 1.4152687788009644, -0.019287319853901863, 0.8243860006332397, -0.02246030606329441, -0.9666715264320374, 1.1805425882339478, 0.8293572664260864, -0.16728883981704712, 0.5508543848991394, -0.3868246078491211, -0.06709689646959305, -0.14762671291828156, 0.4632244408130646, 0.7143086194992065, 0.17975984513759613, 1.3459731340408325, -1.3282300233840942, -0.15448081493377686, -0.6665412783622742, -0.11267439275979996, -0.8615421652793884, 0.2844785749912262, -0.6927122473716736, 0.9051915407180786, 0.8713215589523315, -0.9023340344429016, 0.970533549785614, 0.8427900075912476, -0.5268255472183228, -0.5887680649757385, -0.5604248046875, 0.5973680019378662, 0.007740777917206287, -0.09706221520900726, 1.1437561511993408, -0.04019481688737869, -1.9264540672302246, 0.022647621110081673, -0.7603703141212463, 0.0969814583659172, 1.4939582347869873, 1.164644718170166, 2.4254653453826904, 1.1270346641540527, -0.30245450139045715, -1.0072987079620361, -1.1031557321548462, 1.1571829319000244, -0.38559871912002563, 0.11871088296175003, -0.6794066429138184, -1.074880599975586, 1.3531347513198853, -0.477628231048584, -0.6724964380264282, 1.2096580266952515, 0.460042804479599, 0.414250910282135, -1.4148533344268799, -0.923098623752594, -0.6196255683898926, 0.18302442133426666, -0.20515748858451843, 1.7163504362106323, 0.8787204027175903, 0.48297059535980225, 2.0257010459899902, -0.6192322373390198, 0.511483907699585, -1.9020498991012573, 1.7506155967712402, -0.2117440104484558, -1.0974832773208618, 1.0053117275238037, -0.4575084149837494, -0.12930749356746674, -0.7547958493232727, -0.17216594517230988, -1.9821876287460327, 0.25029629468917847, -0.6048494577407837, -0.8008766770362854, -1.061982274055481, 1.2759103775024414, -0.8367002606391907, 0.9565373063087463, 1.4504950046539307, -1.3858695030212402, 1.5782147645950317, 1.1430294513702393, -0.9396441578865051, 0.19744466245174408, -1.7566838264465332, -0.10928989946842194, -1.1807490587234497, -0.9489592909812927, -0.5794245004653931, 0.22077801823616028, 0.02383730560541153, 0.03974706679582596, 1.0518968105316162, 2.3992056846618652, -0.48459577560424805, -1.55556058883667, 0.8257843255996704, 0.9558677077293396, -0.5333377122879028, -0.6711046695709229, -0.2992265224456787, 1.130362868309021, 0.3994438052177429, -1.2786834239959717, 1.1022874116897583, -1.003920316696167, 2.006887674331665, 0.7193902730941772, -0.29006803035736084, -1.6383693218231201, 1.4087679386138916, 0.3371027112007141, -0.058701056987047195, -0.8012158870697021, -0.00048809440340846777, 0.3454394042491913, 1.047298550605774, -1.8029464483261108, 0.13320903480052948, -0.0598478764295578, 0.24382001161575317, -0.7754408717155457, -0.9677160978317261, -0.20116031169891357, -0.6627413034439087, -0.512645423412323, 0.884523868560791, -1.0483667850494385, 0.2444937825202942, -2.6957781314849854, -0.3529212176799774, 0.4302818775177002, 0.6155116558074951, 0.474294513463974, -1.3185503482818604, -1.1392366886138916, -0.13498473167419434, -0.4112113118171692, 2.0186703205108643, 0.29801860451698303, 0.03687324747443199, -0.32317453622817993, -0.8334805369377136, 1.133628249168396, 2.2083685398101807, 1.6616207361221313, -0.27566763758659363, -1.367448329925537, -0.15366165339946747, 0.7310583591461182, 0.10223767906427383, -0.9233295321464539, -1.8023653030395508, -1.532055139541626, -0.571751058101654, 0.5283808708190918, -1.324587106704712, 0.10039462149143219, 0.5373272895812988, 1.446022629737854, -0.03599967062473297, 1.649912714958191, 0.6202772855758667, -1.3830552101135254, 0.753072202205658, 0.4195863902568817, -0.2152484655380249, -1.3063952922821045, 0.10839932411909103, 0.5756272077560425, 2.0307538509368896, 0.480527400970459, 0.9609947800636292, 1.1149930953979492, 0.8786380887031555, -0.28366079926490784, -0.4207685887813568, -0.730712354183197, 0.2846018075942993, 1.6639611721038818, -0.3636537492275238, 0.2185436338186264, 0.276470422744751, 1.1487102508544922, -1.4794639348983765, 0.8448345065116882, 0.4074796438217163, -1.4574790000915527, 0.03145609050989151, 0.27554404735565186, 0.4903639853000641, 0.39329737424850464, -0.5742872953414917, 0.3668525218963623, -1.6395549774169922, -0.06747133284807205, -0.04713790863752365, -1.0620665550231934, -0.5224742889404297, 0.2900269031524658, -0.6466130018234253, 0.4819111227989197, 0.32893702387809753, 0.26896849274635315, 1.0269254446029663, -1.2111506462097168, -1.308853030204773, 0.9978389739990234, 1.1929949522018433, 0.28327032923698425, -0.3231981694698334, -0.12962891161441803, 0.547996461391449, -0.028742562979459763, 2.146327495574951, -0.009695343673229218, 1.5980520248413086, 0.7331565618515015, -1.3832838535308838, 0.21380853652954102, 0.37767907977104187, 1.2937155961990356, 0.0892874151468277, -0.20360644161701202, -0.08933120220899582, -1.8897736072540283, 0.5767966508865356, -0.644401490688324, 2.072467088699341, -0.44803744554519653, -0.5954318046569824, 1.0315110683441162, 0.7606037259101868, 0.8611006140708923, 0.6317934989929199, -0.19567528367042542, -0.19950363039970398, -0.34491947293281555, -0.7250675559043884, 1.2187716960906982, 2.0552239418029785, -1.1128349304199219, 1.1462111473083496, -0.06228182837367058, 0.7860980033874512, -1.8268778324127197, 1.354738473892212, 0.6269391179084778, 0.01305733434855938, 1.000203013420105, 0.5518520474433899, 0.027018701657652855, 0.331086128950119, -0.26235294342041016, -0.3399486541748047, -0.08633499592542648, 0.22013713419437408, -1.1425000429153442, 0.5144705772399902, -0.07687240839004517, 0.3895849585533142, 0.4651738107204437, 0.6619195342063904, 0.1965833157300949, -0.07859467715024948, 1.104944109916687, 0.788198709487915, 0.12724332511425018, -0.9452454447746277, 0.9286591410636902, 0.4205264747142792, 1.098160743713379, -0.7455878853797913, -0.38522428274154663, 1.4784668684005737, 1.1604175567626953, -0.22947381436824799, 0.03677450865507126, -0.5784608125686646, -1.0111052989959717, -0.7207289338111877, -0.9134767055511475, -0.6692111492156982, -0.24256892502307892, 0.1253276765346527, -0.3873603940010071, -0.37289220094680786, -0.30514034628868103, 3.0157597064971924, -0.6879724860191345, -1.2364753484725952, 0.7390839457511902, 0.8176566362380981, -1.779083013534546, -0.3262508809566498, -0.7527825832366943, 0.09147791564464569, -0.5296739339828491, -1.2242366075515747, 0.7718284130096436, -0.2062762975692749, 0.5270823240280151, 0.01640354096889496, -0.741847038269043, -1.559981346130371, -1.0539016723632812, 0.011731492355465889, -0.3677971065044403, 0.2772826552391052, -0.017252016812562943, 1.0043388605117798, -0.5140770077705383, 0.502751350402832, -0.021612349897623062, 1.7803279161453247, -0.19726134836673737, -0.5324020981788635, 1.415493130683899, -1.2857915163040161, -0.04740238934755325, -1.0441139936447144, 0.013951008208096027, 0.5465916991233826, 0.29645660519599915, 0.0426490381360054, -0.7068814635276794, 0.3945653736591339, -2.42775821685791, 0.0791352167725563, -0.4429062306880951, -0.7279667258262634, 0.3093442916870117, 0.5187769532203674, -0.1923438012599945, 1.5612778663635254, 1.0631256103515625, -1.1880128383636475, -0.33152657747268677, -0.3273880183696747, 0.49537357687950134, -0.719599187374115, -0.3540875017642975, 0.47445252537727356, -1.6808974742889404, -0.2383681982755661, 0.4215346872806549, -0.8967041969299316, -0.753562867641449, -0.9629729390144348, 0.11463335901498795, 0.39507216215133667, -0.3207054138183594, -0.9975879788398743, 0.2995481789112091, 0.03216048330068588, 0.2933523654937744, -1.2001347541809082, -0.43635618686676025, 1.1547590494155884, -0.7249643802642822, 1.608056902885437, 0.6015631556510925, 0.6101788282394409, 0.4650173783302307, -0.11217080801725388, -0.24297180771827698, -0.1559067666530609, 0.08666928857564926, 1.2952165603637695, 0.03257795795798302, 1.7116621732711792, -0.40053749084472656, 0.006014608778059483, -1.015706181526184, -1.0612071752548218, 0.20390139520168304, -0.8265910744667053, -1.2610348463058472, 0.6842599511146545, 0.12404677271842957, 0.1217242181301117, -0.18738403916358948, -0.48786333203315735, 0.8840199112892151, -0.30535802245140076, -0.2593061923980713, -0.26799362897872925, 0.3222997486591339, 0.9980883002281189, 1.2925282716751099, -1.3468023538589478, 0.06871897727251053, 0.8116080164909363, 0.0507647842168808, 0.24222970008850098, 0.9568958878517151, 1.1479394435882568, 0.19162221252918243, 0.162733256816864, -0.11786545068025589, -0.44889259338378906, 0.9834704399108887, 1.5575047731399536, -2.2712385654449463, 0.5791520476341248, -0.16096653044223785, 0.355562299489975, 0.2688658833503723, -0.20242337882518768, 0.5650525689125061, -0.5670836567878723, -0.3141528367996216, 0.2150581330060959, 0.38148993253707886, 1.3252973556518555, 1.135227084159851, 0.11314931511878967, -0.41889098286628723, 2.3419604301452637, 0.06897817552089691, -1.4633480310440063, 1.6571481227874756, -0.8319326043128967, 0.9648960828781128, -1.7100474834442139, -1.244373083114624, 0.057839736342430115, -1.5905905961990356, -0.4797258973121643, -0.31473684310913086, 0.5981014966964722, 0.5361037254333496, 2.178037166595459, 2.1211931705474854, 0.521176815032959, -0.46312907338142395, -0.628380537033081, 0.046929966658353806, -0.09508366882801056, -0.6467449069023132, -0.16123469173908234, 1.7357382774353027, 0.42384809255599976, -0.5038063526153564, 1.6714398860931396, -1.4529365301132202, -1.1612993478775024, 0.8465144038200378, -0.7443884611129761, 0.3218444287776947, -0.24133245646953583, -0.10105295479297638, 1.1619703769683838, 0.9767281413078308, 0.3375853896141052], [-0.636566698551178, -1.1432652473449707, 0.03141340985894203, -1.0533219575881958, -1.5272634029388428, -0.27211087942123413, -0.9721347093582153, -1.765790581703186, 0.6109961867332458, 0.9594532251358032, 1.590585708618164, 1.4677903652191162, -0.21019557118415833, -1.123307466506958, 0.7969942092895508, -0.742954671382904, 1.420919418334961, -0.47290194034576416, 0.4718288481235504, 0.15307332575321198, -0.43519651889801025, -0.41715893149375916, 0.71366947889328, 0.02148880623281002, 1.8999474048614502, 0.42719703912734985, -1.8790642023086548, -0.4500552713871002, 0.8086423277854919, -2.113504409790039, 0.4502421021461487, -0.19392149150371552, 0.08168336749076843, 0.5804713368415833, -1.1642264127731323, -0.3680168092250824, 1.0468757152557373, 0.4042402505874634, -0.24658332765102386, -1.2524056434631348, 1.191219687461853, -1.2168536186218262, -0.6268824934959412, 1.7002955675125122, 1.6128228902816772, -0.22828373312950134, -0.6100185513496399, 1.313653588294983, -0.21842831373214722, -0.24479369819164276, 0.05233991891145706, 0.7178098559379578, -1.0674386024475098, -1.6637932062149048, -0.2762058675289154, -1.3575600385665894, 1.8224440813064575, 0.12188572436571121, 0.25816452503204346, 0.34520846605300903, -1.2819781303405762, 2.4342615604400635, -2.0552473068237305, -1.0545841455459595, 0.17666621506214142, -1.1479026079177856, -0.1417376697063446, 1.014076828956604, -0.3681316673755646, 0.6195352673530579, 0.5344039797782898, -0.5739368796348572, -0.04104248806834221, 0.6291826963424683, 1.0710614919662476, 0.6678078770637512, -0.18258816003799438, -0.06762421876192093, -0.6972662806510925, -0.7434940934181213, 0.7966933250427246, 0.4205527603626251, 0.8544133901596069, -0.5307614207267761, 0.3312871754169464, 1.1245931386947632, 0.7530982494354248, 0.1944640576839447, -1.1266124248504639, -0.7095171213150024, -1.2754790782928467, 1.5569920539855957, -0.6089361906051636, 0.658374547958374, 1.7329431772232056, 0.31330224871635437, 0.9070911407470703, 0.474132239818573, -0.6432222723960876, 1.0272175073623657, -0.9261491298675537, 1.9969722032546997, -0.7737817168235779, 0.9581828713417053, -1.3359888792037964, 0.596320390701294, -0.41062796115875244, 1.095324158668518, -0.5157046318054199, -0.45234623551368713, 0.34839096665382385, -0.3659205138683319, 0.637330174446106, 0.27943578362464905, 1.503868818283081, -0.22956185042858124, 1.0727856159210205, 0.3179669976234436, -0.04764196276664734, 0.5104545950889587, 1.6128690242767334, -1.7792474031448364, -0.5299390554428101, -1.2333056926727295, 0.3338010013103485, -0.3567201793193817, 0.031889576464891434, 0.05043463781476021, -0.0858713760972023, 0.5578703284263611, 0.04183030501008034, -1.2724586725234985, 1.796850323677063, 0.6832241415977478, 0.8914132714271545, 1.2871171236038208, 0.7691394686698914, 0.4357620179653168, 0.6131530404090881, 0.8090023398399353, 0.28525957465171814, -0.048737868666648865, -2.428380012512207, 0.4688085615634918, -0.651441216468811, -0.06711428612470627, -1.3151729106903076, 0.15312157571315765, -1.6264017820358276, -0.061846088618040085, -0.3468199074268341, -0.09189868718385696, -0.06752591580152512, 0.5120841860771179, 0.29080334305763245, -1.021224856376648, 0.191910058259964, -0.9496411085128784, -0.04710117727518082, 1.3886522054672241, 1.2681992053985596, -0.23566600680351257, 0.28611141443252563, -1.4143575429916382, 0.2418561428785324, -0.0006511430838145316, -1.1500232219696045, -0.4729859232902527, -0.14141367375850677, -1.3187752962112427, -1.3339617252349854, 0.4018312692642212, 0.123743437230587, 2.479804039001465, 1.0128103494644165, 0.7319898009300232, -0.9670936465263367, -0.06993380188941956, 0.5480560660362244, -0.9785906672477722, -1.0906023979187012, 1.835037112236023, 0.333118200302124, -0.6523739099502563, -1.8159269094467163, 0.09115032106637955, 1.4072165489196777, -1.5425922870635986, 1.094862461090088, 2.014730930328369, -0.48476091027259827, 0.581928014755249, -0.026928139850497246, 0.019603071734309196, 1.5980358123779297, 0.0713639110326767, -1.7576490640640259, 1.1823455095291138, 0.11311399936676025, 2.0392825603485107, -1.1609786748886108, 1.3267396688461304, 0.4335775673389435, 0.13660579919815063, -0.7089680433273315, 0.04128159582614899, -0.49910181760787964, -1.4501458406448364, -1.0649255514144897, 1.3948818445205688, -0.35524114966392517, 0.6111619472503662, 0.7466607689857483, 0.5245922803878784, -1.4424865245819092, -0.25968217849731445, -1.6397497653961182, 0.9282609224319458, -0.3977831304073334, -0.21480634808540344, -1.4300659894943237, 0.6070340871810913, 0.41750335693359375, 0.8770985007286072, 0.4848285913467407, -1.301011085510254, -0.11584162712097168, -1.1790539026260376, 0.8298463821411133, -0.4618109464645386, 0.7836784720420837, 0.37603551149368286, -0.6253869533538818, 0.8424701690673828, -0.38092565536499023, 0.749213695526123, 1.104418396949768, -0.7115187644958496, 0.2643609046936035, 0.7252376675605774, -0.6466743350028992, 0.9703675508499146, -0.047660503536462784, 0.14341607689857483, -0.0021864434238523245, 1.803466796875, 1.0342659950256348, 0.48704782128334045, 0.27750444412231445, -0.6757547855377197, -0.33448266983032227, -0.1758480668067932, -0.651266872882843, -1.9774339199066162, 0.8449947237968445, 0.6148937344551086, 0.010357359424233437, -1.0677393674850464, -0.8260709047317505, 0.76987624168396, 0.4525844156742096, -0.7625699043273926, 0.17434334754943848, 0.08131898194551468, -0.8006974458694458, 1.139660120010376, 0.12997734546661377, -0.21178971230983734, -1.2145583629608154, -1.1894605159759521, -0.5018132328987122, -0.38716208934783936, 0.658829927444458, -1.4231544733047485, -0.9335429072380066, -2.0903842449188232, -0.2936854362487793, -0.8754433393478394, -0.7463786005973816, -0.1825636774301529, 0.11403464525938034, 0.19798226654529572, 0.44434893131256104, 0.9486569762229919, -1.2260408401489258, 1.4395780563354492, -1.9642846584320068, 1.1786131858825684, 1.4788163900375366, -0.1314781904220581, -2.0476150512695312, 0.8516976237297058, 0.22762437164783478, 0.16875915229320526, -0.42859554290771484, -0.28071436285972595, -0.5879998207092285, -1.5177322626113892, 0.02197239175438881, 0.4508211612701416, -0.7814223170280457, -1.1857175827026367, -0.06706437468528748, -0.42171183228492737, -0.5939464569091797, -0.4229249656200409, 0.7220748662948608, 0.8144421577453613, -0.15591683983802795, 1.270310640335083, 0.11774785816669464, 0.7348760366439819, -0.10963928699493408, -1.249261736869812, 0.9525628685951233, -0.1550489068031311, 0.16565464437007904, 0.04477960616350174, 0.19167454540729523, -0.031519565731287, -0.623237133026123, -0.6478649377822876, 0.031876090914011, -0.21586374938488007, -1.0980136394500732, 0.2254461646080017, -0.4736078083515167, -0.04882868006825447, 1.768387794494629, -1.1023249626159668, 0.7531430721282959, 1.161765217781067, 0.779628336429596, -0.5733880400657654, -0.7489628791809082, 0.7791026830673218, 0.5661414265632629, -0.4660148620605469, 0.21504837274551392, 0.4242933690547943, 2.188584327697754, -1.2405593395233154, -1.3926396369934082, -1.6131186485290527, -0.9719268679618835, -1.2796136140823364, -0.11194659024477005, 0.8301096558570862, -0.9370309710502625, -0.8434173464775085, 0.6632673144340515, -0.5428107976913452, 0.47566065192222595, -0.5822257399559021, 0.992368757724762, 0.2960297167301178, 1.2970359325408936, 0.4528684616088867, 0.16409684717655182, 0.6140832304954529, 1.0137183666229248, -0.19246424734592438, -0.06132527068257332, -0.4152351915836334, 0.968339204788208, -0.19398929178714752, 1.1714986562728882, 0.2160153090953827, 0.11512529104948044, -0.152532696723938, -0.4059029221534729, -1.4305753707885742, 0.30649372935295105, 0.5866736173629761, 1.7950501441955566, 2.3727641105651855, -0.1815641075372696, 0.849599301815033, -0.97975093126297, -0.16766071319580078, 0.3081457018852234, 0.14699362218379974, 2.1334517002105713, -0.4114706814289093, 2.546182632446289, 0.05158865079283714, 0.5024757385253906, -0.5796529650688171, 0.5785543322563171, -2.2589452266693115, -0.5003083944320679, -1.6372401714324951, 0.37905001640319824, -0.1860235184431076, 0.6672467589378357, 0.03709635138511658, -0.051000602543354034, -1.1138978004455566, 0.17464184761047363, 1.0811454057693481, 1.8627288341522217, 0.05320511758327484, 0.312669962644577, 0.4929034411907196, -0.7460861802101135, 0.2671750783920288, -0.9747378826141357, 2.308349370956421, 0.6961469650268555, 0.937489926815033, -1.248583436012268, -1.3413368463516235, -0.32224100828170776, 0.6538026928901672, -1.031293511390686, 2.7481272220611572, -1.8084346055984497, 0.9406991004943848, 0.8239638805389404, -1.1438274383544922, 0.21386227011680603, -0.17461052536964417, 0.7661672234535217, -0.32741421461105347, 0.0008847520803101361, 1.7599669694900513, -0.08425204455852509, 0.08445610105991364, 0.21288372576236725, -0.6826945543289185, 0.6233017444610596, 0.6977972388267517, 0.4579121470451355, 0.3555011749267578, 0.02221496030688286, -1.0161293745040894, 1.0983879566192627, 0.9012942314147949, -2.3305773735046387, -0.2518972158432007, 0.6402084827423096, 0.13574545085430145, 2.8780081272125244, -1.7180052995681763, 0.8837173581123352, 2.266639471054077, 0.22379063069820404, -0.6408177614212036, 0.5707050561904907, -0.7412815690040588, 1.762306571006775, 0.46237456798553467, -0.5942272543907166, 1.0430649518966675, 2.729966402053833, -0.6970937252044678, -0.4414636790752411, -0.9808518886566162, -1.7634549140930176, -0.12547999620437622, -0.9703023433685303, -0.6100624203681946, 1.7344304323196411, -0.8162552118301392, -0.37798652052879333, -1.1261814832687378, 2.3050451278686523, -1.1712795495986938, 0.20552773773670197, 1.8990381956100464, -1.5063889026641846, -1.3193938732147217, -1.9599835872650146, 0.3980151116847992, 0.09441466629505157, -0.30820053815841675, -0.22870568931102753, -1.0922913551330566, 0.8039164543151855, -1.6638368368148804, -2.1604490280151367, -1.336480975151062, -0.5689742565155029, 1.4472769498825073, 0.11624938994646072, -0.4405231773853302, -0.19417189061641693, -0.4045955240726471, -0.2852727472782135, -0.9880069494247437, 0.27564364671707153, -1.3512550592422485, 0.9311472773551941, 0.3944495916366577, -0.8283576965332031, -0.4400794208049774, 0.12448104470968246, 1.0146111249923706, 1.4723381996154785, 1.3654404878616333, 0.17312712967395782, 1.017928957939148, 2.1695542335510254, 1.3502901792526245, 0.190928116440773, -1.618283987045288, 0.21671274304389954, 1.0059478282928467, -1.4110296964645386, 2.1044397354125977, -0.8569580912590027, -1.2005912065505981, 0.8263090252876282, 1.5380516052246094, -1.0255154371261597, 0.06098906323313713, -1.1857903003692627, -0.5960748791694641, -0.0732828825712204, 0.07668069005012512, 0.20156051218509674, -0.30490806698799133, 0.8435284495353699, 0.4131217896938324, -0.6276357173919678, 1.8655999898910522, -0.4677238166332245, 1.1489155292510986, 0.08235380053520203, -0.3114275336265564, 2.381289482116699, 0.01033108588308096, 0.9352928996086121, 0.7562279105186462, -0.14059452712535858, -0.552030622959137, -0.07010746002197266, 0.6762828826904297, 1.6060051918029785, -0.003909572958946228, 0.4008956849575043, -0.6804832220077515, -0.43604740500450134, 0.9148901104927063, 0.2044375091791153, -1.6401368379592896, -0.8309791088104248, -2.6496992111206055, -0.48706328868865967, -1.2312884330749512, 1.7107036113739014, -1.9919345378875732, 0.11408153921365738, 0.22526942193508148, -0.2440214604139328, 0.9549297094345093, -0.22302313148975372, 1.4143680334091187, -0.27198168635368347, 0.2819540500640869, 0.35651707649230957, -1.2345681190490723, -1.581191897392273, -0.44882839918136597, -2.325955629348755, -2.25030779838562, -0.03022128716111183, 0.013113269582390785, 1.5010353326797485, 0.44237881898880005, 0.9074003100395203, 0.8558070063591003, -0.5831862092018127, 0.4602418541908264, 1.1476715803146362, -0.8461033701896667, 0.6284916400909424, 1.126373052597046, -1.131542682647705, 0.15636210143566132, -1.605568289756775, 1.2897319793701172, 0.9239868521690369, -1.907961130142212, 0.4296351373195648, -0.889755368232727, -0.24043451249599457, -0.6434716582298279, 0.3798193335533142, -1.430088996887207, 1.3560909032821655, -0.5858651995658875, 0.7474263906478882, -0.31212252378463745, -1.9913867712020874, 0.47428303956985474, 1.3418514728546143, 0.3240087032318115, 0.9978708624839783, 0.14132380485534668, 0.3265646994113922, 1.2871325016021729, -1.3940590620040894, -1.1391273736953735, -0.4476754665374756, -0.1306823492050171, 0.5163127779960632, 0.21238018572330475, 0.7730020880699158, -0.46538281440734863, -0.8296365141868591, -0.22365997731685638, 0.31832200288772583, -1.1472411155700684, 0.3595150411128998, 0.4544300436973572, 0.47551003098487854, 0.9455301761627197, -0.5597437024116516, -0.9015143513679504, 0.0019266231684014201, 1.080835223197937, 1.1482824087142944, 0.6395359039306641, 0.12729832530021667, -1.9598228931427002, -0.851401150226593, -0.08890451490879059, -0.9398989081382751, -1.1853857040405273, -0.3629722595214844, -1.5291955471038818, -0.047786496579647064, -0.5302436947822571, 0.24600303173065186, -1.457328200340271, -0.11804953217506409, 0.07520335912704468, 1.2790840864181519, 0.20040558278560638, -2.2490665912628174, 0.23066793382167816, -0.3484593629837036, -0.0100669264793396, -0.3252955377101898, 0.5896953344345093, -0.31798940896987915, -0.07234412431716919, 0.040119629353284836, -0.6876985430717468, 0.003766674315556884, 1.3935554027557373, 1.535000205039978, 0.9176723957061768, 1.7745475769042969, 1.6914633512496948, 0.10479945689439774, -0.7981036305427551, -1.0159122943878174, -0.19674858450889587, -0.8712061643600464, 1.2906808853149414, 0.3008482754230499, -0.2683430314064026, -1.3984272480010986, -0.6321991682052612, 0.00010003530769608915, 0.2233375757932663, 1.4051563739776611, -0.9410590529441833, 0.0944051593542099, 0.47794580459594727, 0.5355542898178101, -0.8709883093833923, -1.4905987977981567, 0.20477823913097382, -0.6395144462585449, 1.6126723289489746, -0.21283522248268127, 0.6080923676490784, 0.3939598798751831, -0.5298554301261902, -0.15268248319625854, 1.267706274986267, -0.38250190019607544, 2.3654448986053467, 0.7121996879577637, 0.8535721898078918, 1.2822989225387573, -1.3938366174697876, 0.5870832800865173, -0.6097540855407715, -0.2759339511394501, 1.0415675640106201, -0.47536808252334595, -0.01794034242630005, 1.148290753364563, -0.41036903858184814, -0.6032283902168274, -0.7822390794754028, 0.275572270154953, -0.7282028198242188, -0.3249436318874359, -0.4041944146156311, -0.14846253395080566, -0.5033429265022278, -1.97084379196167, 0.9588579535484314, 0.28194403648376465, 1.16025710105896, 0.8970665335655212, 0.28112515807151794, 0.6311590671539307, -2.2923176288604736, 2.0461742877960205, -0.8034366369247437, 0.9422654509544373, -1.2048922777175903, -0.9303455948829651, 0.5366000533103943, -0.0008987026521936059, -1.052871823310852, -1.1141276359558105, 0.5940396785736084, -1.7169907093048096, -0.9694624543190002, 2.4637417793273926, 0.10156136006116867, 0.09572027623653412, -0.761976420879364, 1.177182674407959, -0.0855809822678566, 0.24298958480358124, -1.1222856044769287, 1.4880777597427368, -0.564597487449646, -1.7613410949707031, -0.9960215091705322, 0.026774611324071884, -0.19404290616512299, 1.9304468631744385, -0.29084089398384094, 0.8065854907035828, 0.721192479133606, -0.13007496297359467, -0.00829242542386055, 1.2559667825698853, 1.1476118564605713, 1.2122907638549805, -0.29114851355552673, -1.605484962463379, -0.02468273602426052, -0.4586811363697052, 0.2333761751651764, 0.6175928711891174, -0.05591708794236183, 0.018242819234728813, 0.4014974534511566, -0.7733123302459717, 2.714773178100586, -0.6962781548500061, 0.34287819266319275, 0.3045234978199005, -0.28695881366729736, 1.077132225036621, 0.413839727640152, -0.23695355653762817, -0.034359995275735855, -0.5954244136810303, 0.7961262464523315, -1.0554466247558594, -0.199401393532753, -0.12175557017326355, -0.13685062527656555, -1.5412639379501343, 0.530639111995697, -0.7911115884780884, -0.18441952764987946, 1.6950273513793945, 0.5882359147071838, 0.14345793426036835, -0.8229069113731384, 1.0043513774871826, 0.4157939553260803, 0.46370068192481995, -0.5504754185676575, 0.0984121635556221, 2.0776500701904297, -0.5314410924911499, 0.13778430223464966, 0.4637303650379181, -1.0086158514022827, -0.4040709137916565, 1.8263602256774902, 1.605543851852417, 1.0111918449401855, 0.5384889841079712, 1.0145623683929443, -0.6096930503845215, -1.6608537435531616, 0.5486346483230591, -0.4553054869174957, 0.38804906606674194, 0.8792441487312317, -2.6734702587127686, -1.0838226079940796, -0.2301684319972992, 0.8770229816436768, -0.5660557746887207, 0.5322317481040955, -0.16015462577342987, -0.9707053899765015, -0.3147263824939728, 0.5334990620613098, -0.7492212653160095, -1.4225128889083862, -1.0361419916152954, 0.38686081767082214, 1.4811161756515503, 0.6842162609100342, 0.14098994433879852, 1.374143362045288, 0.36628109216690063, -0.07275242358446121, -0.07720768451690674, -0.1766444593667984, 1.460400938987732, 0.23071414232254028, -0.5207558870315552, -2.079184055328369, 0.6621537804603577, 1.4243369102478027, -0.5390802025794983, -0.8511410355567932, 0.7594480514526367, 0.5336397290229797, -0.5158659219741821, 1.485243320465088, 1.8657604455947876, -0.34226417541503906, 0.5077449083328247, 0.2950267195701599, -1.7223514318466187, -0.46483704447746277, 0.9812843203544617, -0.01705799251794815, -1.3516907691955566, -1.243179202079773, 0.36077675223350525, -0.7716127634048462, -0.13233540952205658, 0.47888293862342834, -0.24416373670101166, 0.1019648090004921, 0.8605574369430542, 1.6671717166900635, 0.690893292427063, -0.41173112392425537, -2.045053005218506, 1.0478651523590088, 0.6946085095405579, -0.1815570443868637, -0.6223390102386475, -0.1252526342868805, 0.41440844535827637, 0.7071416974067688, 0.22911444306373596, -1.092652678489685, 0.06746365875005722, 0.45639070868492126, -0.7185988426208496, -0.8699195384979248, -1.9335476160049438, 0.1057010293006897, -0.202090784907341, -1.4047634601593018, 0.14279688894748688, 0.0643380656838417, 1.6937661170959473, 0.4243585765361786, -1.508620262145996, -0.49739935994148254, 2.4664995670318604, -1.0142954587936401, -0.811741292476654, 0.6953859329223633, -1.429814338684082, -0.9800700545310974, 0.5072357058525085, 0.12094797939062119, -0.7639631032943726, -0.2828567624092102, 0.16421282291412354, 0.7031494975090027, 0.1694953590631485, 0.7840842604637146, -0.19700101017951965, 0.16007302701473236, 0.2892613112926483, 2.03257417678833, 0.2629877030849457, 0.46104228496551514, 1.970744252204895, -0.8361987471580505, -0.6913687586784363, 0.3262897729873657, -0.12998390197753906, -0.41860806941986084, -0.5307362675666809, -0.5271154642105103, 0.16336622834205627, 0.6429795026779175, 0.5616010427474976, 0.8372492790222168, 0.9518803358078003, -0.08680453151464462, -0.5319728255271912, -0.2672181725502014, -0.7987732887268066, 1.2654263973236084, 0.23269326984882355, 2.283717393875122, -0.42512455582618713, -1.3441542387008667, 1.647457480430603, -0.9991488456726074, 0.15706153213977814, -1.2142267227172852, -0.08059468865394592, -0.4179447591304779, 1.5388307571411133, 0.6069508790969849, -0.918628454208374, -0.5583187341690063, 0.31542348861694336, 0.239599347114563, -0.7756156325340271, -0.5120823383331299, -0.2023467868566513, -0.07961151003837585, 0.9590478539466858, -0.44140496850013733, 0.8108914494514465, -0.1733076572418213, -0.7568956613540649, -0.29944556951522827, 0.6552549600601196, -1.4858736991882324, 0.3032316267490387, -0.2582228183746338, -0.4629429876804352, -1.51683509349823, 0.08781851828098297, -0.20739172399044037, -0.20227675139904022, 0.3453969955444336, 1.4525129795074463, 0.19837719202041626, -2.2418179512023926, -1.8224139213562012, 0.8951544165611267, -0.5722711682319641, -0.37648284435272217, -0.5971355438232422, 1.9741735458374023, 1.8086094856262207, 0.707944929599762, -0.5970093011856079, -0.6062881946563721, 0.5112968683242798, 2.2401318550109863, -0.32775840163230896, -0.6342248320579529, -0.0779981017112732, -0.5968107581138611, 1.2613577842712402, 1.625774621963501, -0.2292967438697815, -1.5357506275177002, 0.4942067265510559, 0.1952919065952301, -0.10417112708091736, 1.1905546188354492, -1.1656869649887085, -1.0362896919250488, -0.5977426171302795, 0.14665108919143677, -0.8566535711288452, 0.956870973110199, 0.16030102968215942, 0.9580875635147095, 1.1581944227218628, 0.20303016901016235, 0.45272499322891235, -0.714470386505127, -0.06054432690143585, 0.3893682062625885, -0.019459610804915428, -0.020060883834958076, 0.19470858573913574, 0.5928844213485718, -0.8968082070350647, -0.6823925971984863, -0.1970261186361313, -0.324442595243454, 1.123119592666626, -0.7009449005126953, 0.8911755084991455, -0.6959370374679565, 1.2294038534164429, -1.399476408958435, -1.378666877746582, 0.09446229040622711, -1.8110153675079346, -2.5040364265441895, -0.6680626273155212, 1.2508176565170288, -0.1207745149731636, -0.2448578178882599, 0.9693363904953003, 0.7816610932350159, 2.1765360832214355, -0.4264548718929291, 0.3715960681438446], [1.3592371940612793, 1.2327790260314941, 0.3150903284549713, 0.15097634494304657, -0.5501189231872559, 0.6833906769752502, 1.8349648714065552, 0.17917965352535248, -1.36931312084198, 0.9865052103996277, -0.7430930733680725, -1.1843619346618652, -0.022414760664105415, -0.3215889036655426, -0.7385947704315186, 0.3256698250770569, 1.522165298461914, -2.226677417755127, 0.9219866991043091, -1.0358760356903076, 0.40504711866378784, -1.0656496286392212, -0.5933623909950256, -0.6395918726921082, 1.0257121324539185, 2.4199283123016357, -0.5710921883583069, -0.23958390951156616, 0.9494504928588867, 0.18652929365634918, 1.7456941604614258, 1.43294095993042, 0.1471877545118332, 1.795639991760254, 1.6699414253234863, -0.24680574238300323, 0.07807334512472153, -0.6311711072921753, 1.3296477794647217, -0.9509168267250061, 2.0572669506073, 0.23372489213943481, 0.5207021832466125, 1.8312299251556396, 1.2561367750167847, 0.08141488581895828, -0.8577880859375, 0.45840147137641907, 2.695380926132202, 0.9991534352302551, 0.8031959533691406, -0.46646881103515625, 0.5020395517349243, 0.6943877935409546, -0.7169761657714844, -1.4184505939483643, 0.6766909956932068, -0.029901213943958282, 1.7668583393096924, 1.4650827646255493, -1.5615206956863403, -0.046371448785066605, -1.1422288417816162, -0.1663234978914261, 1.3383188247680664, -0.943964421749115, -0.33504292368888855, -1.356655240058899, -0.6946479678153992, 1.4103803634643555, 1.0967062711715698, -0.13611683249473572, 1.0097837448120117, 0.49858617782592773, 0.6711209416389465, 0.3008183538913727, -0.12381832301616669, 0.37181830406188965, 0.39169496297836304, 0.6943644881248474, 0.4400506615638733, -1.8511308431625366, 0.5532872080802917, 0.43272340297698975, 0.7663686275482178, -0.3310810327529907, -0.04166397452354431, -0.6692959666252136, -0.7450030446052551, 0.18240416049957275, 1.0999352931976318, -1.3708736896514893, -1.688826322555542, -0.5775739550590515, -0.43837958574295044, -2.618356704711914, 0.3299756348133087, 0.7548077702522278, 0.3910066485404968, -0.15078683197498322, 1.3503214120864868, 0.5438303351402283, 0.9567130208015442, 0.9304661154747009, -1.0186059474945068, 0.7258425354957581, -0.6054741144180298, 1.2802997827529907, -0.3044881224632263, -0.9913221001625061, -1.381991982460022, -0.064263254404068, 2.290391206741333, 0.8579282760620117, 0.6143979430198669, 0.0414225198328495, 1.0412788391113281, -0.860480785369873, -0.3916364312171936, -0.624793529510498, -0.10991386324167252, -0.5960296988487244, -0.24188169836997986, 0.015081742778420448, 0.6985327005386353, 0.8936707973480225, -0.12490817159414291, 0.031124049797654152, 0.5593581795692444, 0.46115052700042725, 1.7052170038223267, -0.43439868092536926, 0.15635943412780762, -1.0876357555389404, -2.2876832485198975, 1.0596375465393066, -0.9503725171089172, 0.5905676484107971, -0.7611340880393982, -0.0006720071542076766, 0.2044544517993927, 0.5914373993873596, 0.40569818019866943, -0.40177714824676514, 0.038323551416397095, 0.4959433674812317, 1.3570330142974854, -0.510148286819458, -1.472385048866272, -0.4848588705062866, -0.0566294863820076, -0.9742165803909302, 0.8403250575065613, -1.3444477319717407, 0.9620402455329895, -1.0607355833053589, -0.06403511762619019, -0.11920791864395142, 0.36151933670043945, -0.0990249514579773, 0.8025777339935303, 0.22899508476257324, 0.6295561194419861, -1.0447893142700195, 0.18933352828025818, -1.0182222127914429, 1.3347758054733276, 1.142130732536316, -0.9021019339561462, -1.2006922960281372, 0.8049110174179077, -0.37565428018569946, 1.0957223176956177, -1.474315881729126, -1.1866029500961304, 2.3041231632232666, 1.1908879280090332, -0.640120267868042, -0.6100626587867737, -0.28222236037254333, -0.2535257637500763, 0.2367408722639084, -0.3160678446292877, -0.3336639106273651, 0.21388846635818481, -0.9290024638175964, 0.09997875243425369, 1.042048692703247, -1.747464656829834, -0.7932530045509338, 1.0769684314727783, -2.326014518737793, -0.8297120332717896, -0.9339566826820374, 0.9267969727516174, 0.5894022583961487, 1.8113263845443726, 0.6081181764602661, -2.0385982990264893, 1.7957518100738525, 0.25626063346862793, 1.0048574209213257, -0.9392849206924438, -0.2900736927986145, 0.7280570268630981, -1.4096578359603882, 1.0258967876434326, -0.45353981852531433, 0.17867960035800934, -0.16992245614528656, 0.996832013130188, 0.9668958187103271, -1.2518495321273804, 0.9384558200836182, -0.31245261430740356, -0.8617281317710876, -0.49393948912620544, 1.7501235008239746, -1.0615049600601196, -0.20230500400066376, 0.28259602189064026, 0.4866839051246643, -0.9631511569023132, 1.1681817770004272, -0.8032742738723755, -0.8202936053276062, -1.5057804584503174, -1.157486915588379, -1.7567538022994995, -0.6672029495239258, 0.25552240014076233, -0.05571915581822395, -0.5103417634963989, -0.4929450750350952, -0.14560440182685852, -0.09187904000282288, -1.2161582708358765, -2.0396316051483154, -0.4499439597129822, 1.1654884815216064, -1.836881160736084, -1.2558698654174805, -0.4270143210887909, 0.1705392301082611, -1.254028081893921, -1.0104875564575195, -1.0940124988555908, 0.14705926179885864, 1.5096765756607056, -0.33917665481567383, -0.6060216426849365, 0.9217875003814697, -0.4069596230983734, -1.2386071681976318, 1.6427974700927734, 1.1640346050262451, 1.7882683277130127, -0.34210923314094543, -0.29278120398521423, -0.44234704971313477, -0.49626779556274414, 1.64316987991333, 0.8683018088340759, -1.0053421258926392, 1.9924240112304688, 1.4554822444915771, 0.69663006067276, -0.2391817718744278, -1.1254072189331055, -0.017663491889834404, -1.3945001363754272, -0.18199414014816284, -1.2526295185089111, -0.4577355980873108, 0.7962091565132141, 1.7375507354736328, -0.08247250318527222, -1.2757456302642822, 1.7990391254425049, 0.07157181948423386, -0.7357956767082214, -1.4437342882156372, 0.2715497314929962, -0.40926623344421387, -0.6101275086402893, 0.5256269574165344, 0.6015181541442871, -0.5891093611717224, -0.4057771861553192, 0.21721449494361877, 0.015769673511385918, 1.4686601161956787, 0.11170228570699692, 1.2628222703933716, 1.8199366331100464, 2.8245224952697754, -1.587019681930542, -0.7742959260940552, -0.5178872346878052, 0.4131569564342499, 0.23661617934703827, 0.3137519359588623, 1.339440107345581, -0.051166534423828125, 1.552527904510498, -0.5990981459617615, 0.9124530553817749, -0.20901648700237274, 0.0702049732208252, 0.2696942389011383, 0.27424076199531555, -0.039534520357847214, 1.7856673002243042, 0.007830619812011719, -0.2162308543920517, -0.4935280382633209, 1.2219197750091553, 1.117621660232544, -0.8729636073112488, -1.3306314945220947, -0.6143832206726074, 0.6656972765922546, 0.3215067386627197, -1.0788142681121826, 0.05750880762934685, 0.9849876761436462, -1.1220883131027222, 0.5606657266616821, 0.8249117732048035, 0.27253761887550354, -1.8803579807281494, -1.3340671062469482, -0.5810733437538147, 0.01609981432557106, 1.6469676494598389, 0.10250146687030792, -1.1807435750961304, -1.4212158918380737, -0.7148769497871399, 0.5340936183929443, 1.764906883239746, -0.5909680724143982, 0.9705150723457336, -1.6547077894210815, -0.9502139091491699, -0.9023217558860779, -0.532389760017395, 0.5725249648094177, 0.24162469804286957, -0.23705005645751953, 0.2584599256515503, -0.6558705568313599, 0.35450300574302673, 3.2346341609954834, 2.111340284347534, -1.1128901243209839, -1.5441522598266602, 0.6503385901451111, -0.42735767364501953, -0.06570420414209366, -0.17558206617832184, -1.1716474294662476, -0.3332148492336273, -0.6368252635002136, -1.4928302764892578, 0.3034106492996216, 0.3987421691417694, 0.4669121503829956, 1.641756296157837, -0.38112834095954895, 0.6274893283843994, 0.4897433817386627, -2.6500399112701416, 1.733304500579834, 1.670738935470581, 0.26231032609939575, 0.34137168526649475, -1.991764783859253, -0.8246083855628967, -0.003783823223784566, -0.4376152455806732, -0.35543015599250793, 1.466666579246521, -2.626039743423462, -0.8674898743629456, -0.1318061202764511, 1.8261876106262207, -1.6357908248901367, 2.224764108657837, 1.2723878622055054, 0.8529940843582153, 0.00534086162224412, -0.3062300682067871, -0.30514317750930786, -1.810162901878357, -0.42942699790000916, 0.7531322836875916, -1.2595499753952026, 0.49860289692878723, 1.1210073232650757, 0.03858720138669014, 0.1869446337223053, -1.2096576690673828, 0.6242081522941589, 0.7517372965812683, -0.6336445808410645, -0.4879012703895569, 1.2272065877914429, -0.8279246091842651, 0.27294012904167175, 1.1495344638824463, -1.0356500148773193, 1.249977469444275, 1.6969993114471436, -0.5685096383094788, 0.5453396439552307, 0.6296588182449341, 1.51044499874115, 0.24256417155265808, 0.007983039133250713, 0.1908784806728363, 0.0027793908957391977, 0.40196794271469116, -0.2026940882205963, -0.2565900981426239, -0.3614550232887268, 0.76537024974823, -0.509148120880127, -1.7111690044403076, -1.8622097969055176, -1.1528958082199097, -0.17489691078662872, -0.35380035638809204, -2.346278667449951, 1.0117449760437012, 0.8271905183792114, -1.3050438165664673, 1.6102092266082764, 0.8084051012992859, -0.09069236367940903, -0.608074963092804, -0.4150358736515045, -2.930065155029297, 1.6594918966293335, 1.0476263761520386, -0.09174396842718124, -0.16885797679424286, -0.13695253431797028, 0.9400976896286011, -0.8881426453590393, -0.663054883480072, 0.4305473864078522, -0.8988887667655945, 0.7742124795913696, 2.8712878227233887, 0.056346792727708817, 0.22596096992492676, 0.4931441843509674, 1.2153147459030151, -1.8402563333511353, -1.1666592359542847, 0.41417062282562256, 1.078155517578125, 0.8580620288848877, 0.7422317862510681, 0.4890930652618408, 0.4278586208820343, 0.14381778240203857, -0.40022218227386475, 0.7774035334587097, 1.8471473455429077, -0.20544065535068512, -1.3270138502120972, 0.45931968092918396, -1.3861851692199707, 0.7964495420455933, -0.26584818959236145, 0.8293207287788391, -1.4169766902923584, 0.22248050570487976, 0.5527492165565491, 0.3058428168296814, 0.307791143655777, -0.910563588142395, 0.5806926488876343, 1.15048086643219, -0.695379376411438, 0.2136700600385666, 0.8518766760826111, 1.1829146146774292, -0.18076427280902863, -0.33737117052078247, 0.5524108409881592, -1.4144434928894043, -0.7943698763847351, -0.20772811770439148, -1.394884705543518, -0.16671913862228394, -0.4431597590446472, -0.062495484948158264, 0.3269563317298889, 0.30550920963287354, 0.6550013422966003, 1.0499491691589355, -0.45100682973861694, -0.8588787913322449, -1.6180100440979004, 1.138306736946106, 0.6915396451950073, 0.8470994234085083, 1.7199164628982544, -0.6018336415290833, 0.15734876692295074, -0.40601617097854614, -0.7537780404090881, -0.0283950325101614, -1.5189036130905151, -0.7380159497261047, -0.5241979360580444, 0.9244866967201233, 1.4501194953918457, -0.8990682363510132, -1.1498771905899048, -0.17362424731254578, 0.23744310438632965, -0.03785787522792816, 1.6035726070404053, -1.1139917373657227, 1.1552380323410034, -0.21930144727230072, 0.28765639662742615, 1.159527063369751, -1.2108298540115356, -0.15986040234565735, 1.3226486444473267, -0.2479829490184784, -1.12998366355896, -0.3807141184806824, 1.0506782531738281, -0.5478265285491943, 1.679568886756897, -0.034506555646657944, -0.5946260690689087, -0.6182997226715088, 0.8553358316421509, 1.170790672302246, 1.2624906301498413, 0.1592327356338501, 2.017519950866699, 0.07010670751333237, -1.0796986818313599, 0.17440086603164673, 0.6013978719711304, 1.274138331413269, 0.26810669898986816, 0.7126837968826294, -1.4492313861846924, -0.09172245860099792, 1.5359944105148315, -1.892037272453308, 0.8804696798324585, 0.09950240701436996, -1.1014533042907715, 0.4548763632774353, -0.8719494342803955, -1.3489476442337036, -0.6709770560264587, 0.3662237524986267, -0.02660127356648445, -0.5486075282096863, -0.41394081711769104, -0.37615323066711426, -0.142426535487175, -1.1738828420639038, 1.0590487718582153, -0.8493549823760986, -0.5080188512802124, -1.0638388395309448, 0.21050506830215454, -2.8310766220092773, 0.8420954346656799, -0.7111548781394958, -0.6160668730735779, 0.21490012109279633, -0.3421947956085205, -0.3951832354068756, 0.9759647250175476, 0.5957444310188293, -0.42201051115989685, 1.6267436742782593, 1.4675054550170898, 1.9279394149780273, 0.647262454032898, 0.22698448598384857, 1.5539172887802124, -1.0601282119750977, 1.018756628036499, -0.4147256016731262, 0.07147173583507538, 0.030150379985570908, -0.10296706855297089, -1.1194870471954346, -1.6322466135025024, 0.5837536454200745, 1.1010597944259644, 0.2801486551761627, -0.05195700377225876, -0.15595126152038574, -0.056369442492723465, 0.3911511301994324, -1.4648576974868774, 0.007343134842813015, 0.5708366632461548, -0.2592339813709259, -0.3722808063030243, -0.39237144589424133, 0.782791018486023, -1.050825595855713, -0.059681978076696396, -0.4825083911418915, 0.1440964788198471, -0.7123906016349792, -1.6716337203979492, 0.7081879377365112, -0.5392508506774902, -0.9293656945228577, -1.8950302600860596, 0.7609685659408569, -1.347243070602417, 0.46650412678718567, -1.2838484048843384, -0.5334039926528931, -0.3998909294605255, -1.130063772201538, -1.0669450759887695, 0.8566668033599854, -1.4045164585113525, 1.194027304649353, 0.6257703900337219, 0.5011560916900635, -1.5471361875534058, -2.057445764541626, 1.0488066673278809, -0.33438342809677124, 1.4420474767684937, 1.8528964519500732, 0.8036913275718689, 0.5898171663284302, -0.3142087161540985, -0.22289879620075226, 1.2410786151885986, 0.41966164112091064, 0.2981258034706116, 3.005095958709717, 0.5891016125679016, -2.4472835063934326, -0.05724991858005524, -0.7298389077186584, -2.884481191635132, -1.029382586479187, -0.28538215160369873, -0.09243953227996826, 0.8941485285758972, -0.4887904226779938, -0.48813074827194214, -0.4829173684120178, -0.07314188778400421, -1.3244118690490723, 1.3324724435806274, -0.028725871816277504, -0.624789297580719, -0.1820901483297348, 0.2536145746707916, 0.8527112007141113, 0.5401297211647034, 0.22133846580982208, -0.5082504153251648, -1.1194078922271729, -0.3143913745880127, -0.16274482011795044, -0.5363249182701111, 0.8526475429534912, 0.24842569231987, 1.3172837495803833, -0.3433776795864105, 0.8106545805931091, 0.01812664419412613, -1.0948059558868408, 0.9429311156272888, -1.5089597702026367, 1.3630315065383911, -0.009504641406238079, 1.137916922569275, -2.4058170318603516, 0.5900413393974304, -1.2184535264968872, 1.2304174900054932, -0.36664342880249023, 0.769755482673645, -1.7112988233566284, -0.8237395286560059, -0.7924145460128784, -1.5668647289276123, -0.36705437302589417, -0.3960218131542206, 0.9389920830726624, -0.15534107387065887, -0.8430007100105286, -1.9687579870224, -0.5986584424972534, -0.08877544850111008, 0.5201095342636108, 1.8395297527313232, -2.4371659755706787, -0.3891378343105316, -1.0695244073867798, -0.01494250912219286, 1.0898733139038086, 0.015021681785583496, -0.7858782410621643, 0.26329436898231506, 0.008521531708538532, 1.0058298110961914, -0.9719415903091431, -1.4592390060424805, 0.2507359981536865, 0.6747614741325378, 0.7942391633987427, 0.5991561412811279, -0.8140949606895447, -0.4021989703178406, -0.342290997505188, 0.024214042350649834, -1.14805269241333, -0.8369696736335754, -1.053939700126648, -1.3711214065551758, -0.028772907331585884, -1.1899778842926025, 0.822909951210022, -1.0225716829299927, -0.024607239291071892, -0.7995041012763977, 2.6329543590545654, 0.949989914894104, 0.9804137349128723, -0.5148196220397949, -0.3472338616847992, 1.2933123111724854, -0.639457643032074, 0.4125133156776428, 0.35724306106567383, 0.5282047390937805, 3.234234571456909, -0.7782414555549622, -1.3247649669647217, -0.4632488787174225, 0.42661336064338684, -1.0174163579940796, 0.7791397571563721, -0.3331787586212158, 0.6087359189987183, 0.23922690749168396, 2.41548228263855, 1.7862871885299683, -0.44355955719947815, -0.5891802906990051, -0.22722283005714417, -0.004756918177008629, -0.030419498682022095, 0.2773900330066681, 0.9789392352104187, 1.1972190141677856, -0.4223494529724121, 0.4540054500102997, -0.4907611906528473, 1.8343720436096191, 0.5971317887306213, -1.4907172918319702, 0.44848427176475525, 0.4343641400337219, -1.2288177013397217, -1.2968366146087646, -1.324769377708435, 0.806590735912323, 0.14929652214050293, 1.7497962713241577, 0.2653505802154541, 0.23881472647190094, 1.8788750171661377, 0.6206038594245911, -0.03166459873318672, -0.05703664943575859, 1.2230727672576904, 1.725713849067688, 0.42111584544181824, -0.19360464811325073, 1.105056643486023, 1.0695903301239014, -0.05835604667663574, -1.1914739608764648, -0.2702440321445465, -0.8722221851348877, 1.2063624858856201, -1.2352368831634521, 1.2058829069137573, 3.3617656230926514, -0.3783613443374634, -0.9684282541275024, 1.121983289718628, -0.04961797595024109, -0.8751207590103149, 0.3958911597728729, 0.04414549842476845, -0.6733987927436829, -1.1919069290161133, 0.3042701780796051, -0.13066717982292175, 0.05419781804084778, 0.6107765436172485, 1.0527430772781372, -0.49374303221702576, 3.575380563735962, -1.8742034435272217, 0.5553447604179382, -0.41477593779563904, 0.3495439887046814, -0.2893546223640442, 1.3008062839508057, -1.4664894342422485, -1.539359450340271, 1.5281914472579956, 2.5043818950653076, -0.4769628643989563, -0.11041906476020813, 0.5793914198875427, 0.41804808378219604, -1.324583888053894, 0.7739684581756592, 0.3077811896800995, 0.8160744309425354, 0.053793467581272125, -0.2736888825893402, -0.8132449388504028, -0.953239381313324, 0.8546447157859802, -0.586726188659668, 0.8127784132957458, -1.3482542037963867, -0.7129548192024231, -1.3442994356155396, -0.8782537579536438, 2.035980701446533, -0.2931494116783142, 1.4828275442123413, 0.8100464940071106, -1.0441495180130005, -0.49543121457099915, 1.0557265281677246, -0.15162014961242676, -0.24796029925346375, -1.0092977285385132, 1.3145848512649536, -2.032926559448242, -0.22243592143058777, 0.06343324482440948, -0.4991309344768524, 1.3396859169006348, 0.3650733530521393, -1.3487470149993896, 0.17245818674564362, -0.4040607511997223, -1.0280183553695679, 1.6025789976119995, -0.7845537662506104, -1.4573861360549927, -0.5075563788414001, 0.19929684698581696, -0.8574190735816956, -0.16751553118228912, 0.8504114747047424, -0.6750791072845459, 1.7724721431732178, 0.8615015745162964, -0.42826998233795166, -1.4135196208953857, 2.078188896179199, -0.027083508670330048, 0.32511770725250244, 0.5645391345024109, 0.8800110816955566, 1.0813000202178955, -0.12309307605028152, -1.5363636016845703, 1.1558431386947632, -0.21422839164733887, 0.5938422679901123, -0.6620784401893616, -1.016082525253296, 0.3274850845336914, -0.0714135393500328, 0.13162793219089508, 0.6924341320991516, -0.16233786940574646, 0.9175270199775696, -1.394604206085205, 0.6201368570327759, 0.35148677229881287, 1.4098691940307617, -0.5133153200149536, 1.743727207183838, 0.826872706413269, -0.03593206778168678, -1.5528138875961304, 0.4965818226337433, 0.7685560584068298, 1.5117731094360352, -1.0237921476364136, 0.5707796812057495, -0.9555807709693909, -0.45180201530456543, 0.09390678256750107, -0.5733908414840698, 0.9630599617958069, -0.5708711743354797, -1.9484994411468506, -0.5430924296379089, 1.2710739374160767, -0.880108118057251, 1.5421488285064697, -0.1255026012659073, -0.8038536310195923, -0.3622701168060303, -0.6239482164382935, -1.3746821880340576, 0.08196734637022018, 0.03699566796422005, 0.6875638961791992, -0.26010647416114807, 0.39987561106681824, -0.5296059846878052, 0.30148085951805115, 0.8094478249549866, -0.31643128395080566, 0.47453272342681885, 0.40832874178886414, -0.8755719065666199, 0.9292833805084229, 0.6846520304679871, 1.5201314687728882, 1.8577018976211548, -0.01369626447558403, 0.2640124559402466, 0.24846647679805756, -1.1117594242095947, -0.2600609064102173, 1.1944236755371094, 0.37593764066696167, -1.5548456907272339, -1.422379493713379, -0.03959259018301964, 1.2197983264923096, -0.8924203515052795, 0.39625442028045654, 0.6630238890647888, 0.8757914900779724, -1.9253534078598022, -0.8109280467033386, -0.7323431968688965, 1.0801054239273071, 0.028817446902394295, -2.7836406230926514, 1.9801737070083618, 0.9482877254486084, -0.1404133290052414, 0.16512519121170044, -1.3575488328933716, -1.3830596208572388, 0.7297441363334656, -0.32257696986198425, -0.7540255188941956, -1.9323230981826782, 0.7559201121330261, -1.3869450092315674, 0.4701148569583893, 2.204101085662842, -0.2241334766149521, 0.029802704229950905, 0.2676389515399933, 0.07429055869579315, -1.5820672512054443, 1.343964695930481, -0.3838876187801361, -0.13916997611522675, 1.7438015937805176, 0.1764751374721527, -1.1334075927734375, -0.7169414758682251, -0.26068180799484253, 1.9409064054489136, 1.0727972984313965, -1.9188112020492554, 0.49086442589759827, -0.5226554274559021, 1.846483588218689, 1.3554617166519165, -0.37348711490631104, -0.40530166029930115, -0.36996549367904663, -1.0283634662628174, -0.27042993903160095, -0.23194000124931335, -2.5881295204162598, -0.09290916472673416, 1.8223432302474976, -1.7910629510879517, 1.8341625928878784, 0.20634858310222626, 0.9968981146812439, -0.060418274253606796, 0.2912076413631439, -1.348435401916504, -1.7203772068023682, -0.12904952466487885], [1.0857839584350586, -0.28122836351394653, -0.016599595546722412, -1.7671830654144287, -1.6914035081863403, 0.4189409911632538, -0.3235316872596741, 1.4140753746032715, 0.04710109904408455, -1.1511757373809814, 1.009408950805664, 1.1231646537780762, -0.017250942066311836, -1.028059482574463, 0.8836129903793335, -0.7247598171234131, 0.07939201593399048, -2.6990702152252197, 0.8731416463851929, -0.8573716282844543, 0.5872358679771423, -0.5936768651008606, -0.8109133839607239, 0.19176559150218964, -0.4225523769855499, -0.2191091775894165, 1.320865273475647, 0.509856104850769, 0.748546302318573, -2.1021673679351807, -0.47187089920043945, 1.6835532188415527, -0.6657362580299377, 1.3211594820022583, 1.3304461240768433, -0.29507508873939514, 0.588090181350708, 0.5830315351486206, 2.373469352722168, -0.8385637402534485, 0.29662275314331055, 0.6759243607521057, 0.8662081360816956, 0.711076557636261, -1.316021203994751, -0.8260673880577087, -1.02150559425354, 1.5577877759933472, -0.917169988155365, 1.26223623752594, 0.8444459438323975, -0.8514260053634644, 1.69611656665802, 0.34394708275794983, -1.0984255075454712, 0.3689378798007965, -0.7152199149131775, -0.03344457224011421, -1.893903374671936, 0.12659375369548798, 0.6889103651046753, 0.22579994797706604, -0.5511659383773804, 0.4745580852031708, 0.37189218401908875, 1.3827807903289795, -0.05611686035990715, -0.8771323561668396, 0.06771384179592133, -0.830937922000885, -0.36666780710220337, 0.20599037408828735, 1.0653835535049438, -0.10657915472984314, 0.32798561453819275, -0.36151641607284546, -1.1171094179153442, 0.5385745763778687, 0.04011370986700058, -0.40223732590675354, -0.4318649470806122, 2.307170867919922, 0.6419596672058105, -1.8429585695266724, 0.912638783454895, -0.5265427827835083, 2.214322090148926, -0.1663450300693512, -1.4672456979751587, 0.6363197565078735, 0.8696059584617615, -0.44350665807724, 1.0091073513031006, -0.26730650663375854, -0.3806860148906708, -0.6464534997940063, 0.0768820196390152, -2.300790309906006, 0.11663126200437546, 0.6173167824745178, 0.07579655200242996, -0.34589800238609314, 0.19469432532787323, 0.40491995215415955, 0.6109070777893066, 0.44459035992622375, 0.9212722182273865, 1.981841802597046, -0.24945896863937378, 0.4771574139595032, 0.8914234638214111, -1.3119484186172485, 1.3573061227798462, -0.1906503289937973, -2.03070068359375, -0.7689819931983948, 1.1913173198699951, 0.1784260869026184, -1.3973023891448975, 0.6713846921920776, 0.38749265670776367, -0.5342394709587097, 1.172432541847229, -2.6205854415893555, -0.24471716582775116, 0.6303151845932007, 1.6607011556625366, 0.06581088900566101, -0.44850897789001465, 1.4002039432525635, -0.597369909286499, 0.023124340921640396, 1.0210589170455933, 2.1121625900268555, -0.2691590487957001, -0.2951059341430664, -0.05458793416619301, 1.1189137697219849, 2.1422035694122314, 0.14836381375789642, -0.5064356923103333, -0.2062033712863922, 0.2585894763469696, 0.15058071911334991, 0.9798496961593628, 0.08086921274662018, -0.45957663655281067, -1.0638355016708374, -0.5731226205825806, -0.12997061014175415, -0.4565538167953491, 0.2974751889705658, 2.1365103721618652, -0.13125856220722198, -1.0721060037612915, -0.012405655346810818, -0.48205432295799255, -0.27067384123802185, -0.8657942414283752, 1.1340844631195068, -0.21790538728237152, -0.15831919014453888, -1.2918360233306885, 0.13326245546340942, 0.6937384605407715, -1.0954912900924683, 0.09919042885303497, 0.6249297261238098, -0.6909679174423218, 0.7390885353088379, 0.5092841982841492, -0.3504069745540619, -0.12632444500923157, 0.17449341714382172, -0.7199748158454895, -0.8951702117919922, -0.9316783547401428, 0.1699642390012741, -0.043754346668720245, 0.6738077402114868, 1.046724557876587, 0.14117491245269775, 0.36568892002105713, 0.22856426239013672, -0.9085256457328796, -0.27560168504714966, -2.2138941287994385, 0.7883228063583374, -1.2619032859802246, -0.36728617548942566, -0.6894838809967041, 0.40874090790748596, 0.5982676148414612, 0.09698383510112762, 0.24453061819076538, -0.6686773896217346, 1.209269642829895, 0.0001719899446470663, -0.7789867520332336, 0.3590523600578308, 1.8345308303833008, 1.195818543434143, -1.771428108215332, -0.7481250166893005, 0.7077385783195496, -1.4112168550491333, -0.5223087072372437, 1.5209685564041138, 0.42010265588760376, -0.19392232596874237, -1.032605528831482, 1.3203895092010498, 1.0804513692855835, -0.6225665211677551, -1.2610557079315186, 0.6584818959236145, 2.3235433101654053, 1.9458423852920532, -0.45952537655830383, -0.3809601068496704, 1.8964277505874634, -0.9449068903923035, 1.1135274171829224, 0.7198047041893005, 0.7826707363128662, -0.39877429604530334, 0.28882381319999695, -1.0899579524993896, -0.840668261051178, -0.5524956583976746, -0.17396707832813263, -0.5992189645767212, 1.0181137323379517, -0.6285196542739868, 0.4925384521484375, 2.190502643585205, 0.18606501817703247, 0.30895593762397766, -1.1269152164459229, -0.39336100220680237, 1.9570611715316772, 0.7463873028755188, 1.3611000776290894, 1.3738292455673218, 0.00015352570335380733, -0.8460545539855957, -1.9918544292449951, -0.3091292679309845, 0.736230731010437, -0.3040791451931, 0.43876487016677856, -0.8453050255775452, 0.23822221159934998, -0.48169654607772827, 0.4023626148700714, 0.15995720028877258, 0.8009471297264099, -0.8405787944793701, -1.2078666687011719, 1.1522365808486938, 1.710845947265625, 1.6315178871154785, 0.9365167021751404, 0.6203441619873047, 0.22693635523319244, -0.36740976572036743, 0.9701671600341797, -1.8035732507705688, -0.6615498065948486, 0.5796893835067749, 0.1470886468887329, -1.8346847295761108, -0.7599965333938599, 0.25074636936187744, -0.17290791869163513, 1.9209718704223633, -1.3859745264053345, -0.12584514915943146, -0.5162573456764221, -1.5497078895568848, 0.9588634371757507, 1.485378384590149, -3.274728775024414, -0.48974111676216125, -0.5723612904548645, -0.9513660669326782, 0.4164882004261017, -0.18750715255737305, 0.6383039355278015, 0.156524658203125, -0.8039048314094543, -0.33802399039268494, -0.03675878420472145, -0.1352655440568924, -0.6325342655181885, 0.42824679613113403, 1.0083224773406982, -1.5566277503967285, -0.47733452916145325, 0.37761610746383667, -0.7936793565750122, -0.6288371086120605, 1.3367204666137695, 0.6208430528640747, 0.2853088080883026, -0.6220145225524902, -1.1491647958755493, -0.4331646263599396, -1.4750797748565674, 1.857815146446228, 0.5275474190711975, 0.44782406091690063, 0.7141362428665161, -1.3211023807525635, 0.441253662109375, -2.075507402420044, -1.0873404741287231, 0.7393547892570496, 1.1608383655548096, -1.5123050212860107, 1.3853280544281006, -1.8279379606246948, 0.37462687492370605, -0.5089088678359985, 0.10735703259706497, -1.2432423830032349, -1.9614709615707397, 0.5390284061431885, -0.2359265685081482, -0.9339385628700256, -0.5309332013130188, -0.3568987548351288, 0.18591533601284027, -1.6853033304214478, 0.7176180481910706, 0.19612474739551544, 0.5885409116744995, -0.2765919864177704, 0.1518021821975708, 0.09907558560371399, -0.12074048817157745, 0.840645968914032, -0.41443943977355957, 0.2039688676595688, -0.9082391858100891, 0.6645862460136414, 0.06237762048840523, -1.0008337497711182, -0.18376140296459198, -0.807413637638092, 1.2517036199569702, 0.7471782565116882, -0.052579108625650406, -1.3897336721420288, -0.09116248786449432, -1.6184886693954468, -0.9907160401344299, -0.3082112669944763, -0.5842475295066833, -0.9004901051521301, 1.639372706413269, -0.659880518913269, -1.12149977684021, 1.2391247749328613, 0.9783858060836792, 0.9561367034912109, -0.55620938539505, 0.7758100032806396, -0.9019858837127686, 0.4844924509525299, -0.09325399249792099, -0.9406081438064575, -0.49046817421913147, 0.9778918027877808, -0.3285405933856964, 0.19130532443523407, -0.8859228491783142, 1.2278168201446533, 1.0067440271377563, -0.18136507272720337, 0.9543858766555786, 0.6941283345222473, -1.2119468450546265, -0.2500596046447754, -2.197150230407715, 0.766849160194397, -1.4918971061706543, -2.471240520477295, -1.1768701076507568, 1.1829952001571655, 0.38391977548599243, -1.0151925086975098, -0.22646184265613556, -0.3464510440826416, -2.4889068603515625, 0.7282558083534241, -0.3047267198562622, -1.3990839719772339, 0.05645192787051201, 1.0195276737213135, 0.7939079999923706, 1.52723228931427, -0.3307444751262665, 2.276829481124878, 0.1843539923429489, -1.0369482040405273, 0.5604187846183777, -0.4635988473892212, -0.5355775952339172, 1.9796299934387207, -0.715644359588623, -2.1784324645996094, 0.3297431468963623, -0.9122037291526794, 0.5665891766548157, -0.8527091145515442, 0.804004967212677, 0.27465665340423584, 0.7498103380203247, -0.1078009158372879, -0.3563588559627533, -0.18277627229690552, 0.9043809771537781, -3.5247480869293213, 1.0705269575119019, -0.8694261312484741, -1.044653058052063, -0.4551887810230255, 0.12060566246509552, -1.3713164329528809, -0.6829962730407715, -0.08603505790233612, 1.723852276802063, -0.2982332408428192, 0.5200005173683167, -0.8861216306686401, 0.16949936747550964, 0.412661075592041, -1.192167043685913, 0.33572396636009216, 1.2912354469299316, 1.7169939279556274, -0.9614133834838867, -0.6760784983634949, 1.3576929569244385, 1.2821087837219238, 1.2995080947875977, 0.16168148815631866, -0.40826526284217834, -0.183033749461174, -0.6127818822860718, -0.4239365756511688, 0.9944647550582886, 0.8252983689308167, 0.09146558493375778, -0.8127321600914001, 1.134169340133667, 0.1986607164144516, 1.0929051637649536, -0.5615596771240234, 1.6413816213607788, -2.0491292476654053, 0.3322700262069702, -1.3599369525909424, -1.9079612493515015, 2.2795612812042236, 0.7574238181114197, -1.5240153074264526, 0.2742120027542114, 0.7572408318519592, -0.55950927734375, -0.09010575711727142, 0.8185131549835205, 0.012574822641909122, 0.31492918729782104, -1.9454137086868286, -0.5007082223892212, 0.045186907052993774, 0.3413448929786682, 0.8266754746437073, -0.659654974937439, -0.25828585028648376, 0.6584072709083557, -0.08049295097589493, -0.8884184956550598, 0.3562804162502289, 0.3742177486419678, 3.1177706718444824, 0.6966452598571777, 0.1995198279619217, 1.3817198276519775, -0.37754207849502563, 1.1769150495529175, 2.521266460418701, 0.3521009683609009, 0.6269640326499939, -0.29091084003448486, 2.0835628509521484, 0.7056719064712524, 0.8650547862052917, 0.8498409390449524, -0.7608514428138733, -0.007984375581145287, -0.7363152503967285, -0.6342942118644714, -3.2255752086639404, 0.2831917405128479, 1.1140505075454712, -0.16666990518569946, 0.0809946283698082, 1.6942752599716187, 1.8184527158737183, 1.6835083961486816, -0.08437728136777878, 0.609767735004425, -0.38977935910224915, -0.5696173906326294, -0.33253419399261475, -1.106817603111267, 0.7238912582397461, 0.22856193780899048, -0.9433851838111877, -0.051917415112257004, -1.5845249891281128, -1.764453411102295, -0.5027079582214355, -1.3647286891937256, 0.03820349648594856, 1.2323529720306396, 0.3476001024246216, 0.24602743983268738, -0.14150160551071167, 0.4077984392642975, -1.7564923763275146, 1.5380828380584717, -0.4396193027496338, -1.6322879791259766, 1.7260738611221313, 0.13268254697322845, -1.205295443534851, 1.6259909868240356, -1.5835689306259155, -0.8854458928108215, 0.5930041074752808, 0.96867835521698, 1.8598145246505737, 0.5091460943222046, 1.9568021297454834, 1.4226477146148682, -0.9508348703384399, -0.9752438068389893, 0.7464022636413574, -0.5362532734870911, 0.5437054634094238, 1.6488871574401855, -0.516691267490387, -0.7762178778648376, -0.1733470857143402, 1.0291662216186523, -1.8792465925216675, 0.6943426728248596, -0.3222857713699341, -0.37019646167755127, -0.510913610458374, 0.621891438961029, -2.3203160762786865, 0.5717520713806152, 1.0909104347229004, 0.40081456303596497, 0.2677331566810608, -2.1410961151123047, 0.20750422775745392, -0.5557481050491333, 0.8958218097686768, 1.3845398426055908, -0.4841410517692566, -0.24374744296073914, 1.3974783420562744, 1.1589006185531616, -0.830318033695221, -0.3386843502521515, 0.5251225829124451, 0.3634110391139984, -1.2600394487380981, -1.1540824174880981, -0.06069859489798546, 0.690483808517456, 1.1050890684127808, 0.1723136156797409, -1.6146364212036133, -0.3490101397037506, -0.2883544862270355, 0.27303391695022583, 0.07661991566419601, -0.581516683101654, -0.21186329424381256, 0.25356248021125793, 0.8009955286979675, -1.598294973373413, 0.6099730134010315, -1.1885617971420288, 0.5277003645896912, 1.4188103675842285, -0.6345391869544983, 0.2555471658706665, -0.4223192632198334, 0.12887950241565704, 0.8894500136375427, -0.8405243158340454, -0.010908622294664383, -0.8845529556274414, -0.33215755224227905, 0.30007404088974, 1.2732675075531006, -1.3288544416427612, 1.213423252105713, -1.2104910612106323, 1.663483738899231, 1.0938180685043335, 0.7028481960296631, -0.14601343870162964, 0.2948930561542511, -0.05177168548107147, -0.7737677097320557, 0.6084591150283813, 0.25371649861335754, 1.8494839668273926, 1.996340036392212, -0.9515770673751831, -0.1269201636314392, -0.6058254241943359, -0.2243935614824295, 1.4755761623382568, 0.24478088319301605, 0.07173298299312592, 0.5309009552001953, -1.0713921785354614, 0.6412004232406616, -0.6031501889228821, -1.6681090593338013, -0.05330123007297516, -1.364306926727295, 0.11083628982305527, -0.4016178548336029, -0.4504013657569885, 1.3948155641555786, -0.7387046217918396, 1.3304476737976074, -0.6347476243972778, 0.31166037917137146, -1.7687739133834839, 0.5519419312477112, 1.0321999788284302, -0.7217203974723816, -0.5223792195320129, -0.150634303689003, 1.1476476192474365, 2.0006895065307617, 0.18863707780838013, 1.6038378477096558, 1.3074493408203125, -0.7230612635612488, 0.8220011591911316, 0.45611557364463806, 2.1974103450775146, 0.035070374608039856, -2.197661876678467, 0.2653646171092987, -0.4671133756637573, 1.8870047330856323, -1.0209033489227295, -1.1543433666229248, 0.21681049466133118, 1.0206228494644165, -1.5035808086395264, -1.9392931461334229, 0.014216795563697815, 0.7142786383628845, 0.9839949607849121, -0.314161092042923, 0.5221561193466187, 1.3339807987213135, -1.5289584398269653, 0.04675246402621269, 0.4768218696117401, 3.231198310852051, -0.3479159474372864, 0.4296797513961792, 2.4380991458892822, -0.5754019618034363, 0.03671804070472717, 0.6301166415214539, -2.0169951915740967, 0.41544753313064575, -1.128833532333374, 1.2888398170471191, 0.9786326885223389, -0.03955870121717453, -1.7161386013031006, -0.5440643429756165, -0.3918810784816742, 0.2099699229001999, -2.8201961517333984, 0.9377344846725464, 1.2649050951004028, -1.7666093111038208, -0.4743627607822418, -1.3615660667419434, 0.29230421781539917, -0.8547306656837463, -0.44969698786735535, -0.4982147216796875, -2.0154640674591064, 1.425243854522705, 0.4496276080608368, -2.557814598083496, 0.38665539026260376, 0.5647469758987427, 0.2439858615398407, -0.34182730317115784, 0.9887261986732483, -0.9274221658706665, 0.03188532218337059, -0.14874130487442017, -0.1385691612958908, 0.08396192640066147, 0.8590903282165527, 0.5348437428474426, -1.4334700107574463, -0.9974640607833862, 1.895765781402588, -0.6885640025138855, 1.0836979150772095, -0.6979955434799194, -0.37578722834587097, -0.600665807723999, 1.2816261053085327, -0.7872813940048218, 0.8444341421127319, 0.8586661219596863, -0.7281635999679565, 1.1241199970245361, -1.5319215059280396, -0.44558992981910706, -0.4657285809516907, 0.42119327187538147, -0.6399553418159485, -1.53182852268219, -0.7037197351455688, 0.7929858565330505, 1.6034855842590332, 1.0746846199035645, 1.7907928228378296, 0.9754806756973267, 0.38091936707496643, 0.4355187714099884, -0.1372823715209961, 0.9190091490745544, -1.31264066696167, -1.665298342704773, -1.3361927270889282, -0.7359662652015686, -0.9669598937034607, 1.2675864696502686, -0.8212175369262695, -0.6655570864677429, 1.048699975013733, 0.7608577609062195, 0.3912757337093353, 2.2021048069000244, -1.2478431463241577, 0.4465813934803009, 1.0718516111373901, -1.014636754989624, -0.4707728326320648, 0.5925081372261047, -0.5701542496681213, -0.06052520498633385, -0.024234578013420105, -1.1524291038513184, -0.28015291690826416, 1.0864404439926147, -0.17872552573680878, 0.6508959531784058, -0.9545599222183228, -0.692134439945221, -2.2574987411499023, 0.1273382008075714, -0.731667160987854, -2.323362350463867, -0.07137264311313629, -0.4491240382194519, -0.28873902559280396, 1.497367024421692, -0.5203359723091125, 0.43506231904029846, -0.596329391002655, 1.959486961364746, -0.5181693434715271, 1.4326764345169067, -0.5545846819877625, -0.5777496695518494, -0.0907675251364708, -0.95152747631073, -0.38529035449028015, -0.4238184988498688, -0.8913825750350952, 1.1243031024932861, -0.5674682855606079, 2.3595805168151855, -0.11270981281995773, 0.45491424202919006, 1.5926777124404907, -0.545879065990448, -2.4309237003326416, -1.337889313697815, -0.7410151958465576, -0.4673443138599396, -0.9943076968193054, -0.2901930809020996, 0.6126791834831238, -1.150361180305481, -0.2997633218765259, -0.26696333289146423, -0.3648942708969116, 1.5502277612686157, -0.6389199495315552, -1.5312021970748901, 1.1273361444473267, -0.8498650193214417, -0.021848317235708237, -1.1309936046600342, 1.5997161865234375, 2.501695156097412, -0.69801265001297, -0.8994830250740051, 0.5709357857704163, 0.311160147190094, -0.5238755345344543, -1.9200717210769653, -0.36784207820892334, 1.3429548740386963, 0.7983304858207703, 0.14552530646324158, 0.9506197571754456, -0.5522516965866089, 1.2377482652664185, -1.3772974014282227, -1.1848708391189575, -0.2611110508441925, 1.8564079999923706, 0.18419083952903748, -0.46131423115730286, -0.4979468584060669, 0.06756015121936798, -0.6734697818756104, 0.020435689017176628, 0.07645940035581589, -0.31701087951660156, -1.0982086658477783, 1.644500732421875, 1.4282406568527222, 0.764926552772522, 1.1572856903076172, 1.4997773170471191, -0.292296439409256, 1.0660284757614136, 0.16617697477340698, 0.9270750880241394, -0.9358863234519958, 1.636111855506897, 0.705341100692749, -0.44258013367652893, 0.5463708639144897, -1.3331435918807983, 0.7339402437210083, 1.3083274364471436, -0.464729368686676, 0.6338019371032715, -0.5521149039268494, -1.7628719806671143, 0.45856693387031555, 1.4861165285110474, -0.3578096628189087, 0.407685250043869, 0.5302624702453613, 1.0528740882873535, -0.31184133887290955, 1.811572790145874, -0.9581331014633179, 0.3827543258666992, -0.3529934883117676, 1.2329617738723755, -1.3825823068618774, 1.3233659267425537, -2.291283369064331, -0.9014959931373596, 0.7712859511375427, 1.970361351966858, 0.8821626305580139, -0.4979107677936554, 1.4355653524398804, -0.6982858777046204, 0.6115518808364868, -0.2677125334739685, -0.8886431455612183, -2.1477341651916504, 1.779894232749939, 0.545671284198761, -0.506557285785675, 0.003886308055371046, -0.6816537976264954, 0.2991512715816498, -0.3522297441959381, 0.7863478064537048, 0.5095164179801941, -0.3832557499408722, 0.7469133734703064, -0.48955047130584717, -0.777321994304657, -0.39267030358314514, 2.39845871925354, -0.1070827767252922, -1.8967093229293823, -0.19957345724105835, 0.1251148283481598, -0.532803475856781, 1.3504579067230225, 0.022801119834184647, 1.3103139400482178, -1.1552764177322388, -1.082401990890503, 0.24744024872779846, 1.7115994691848755, 0.14940202236175537, -1.0711824893951416, 1.892106294631958, -0.9030651450157166, -1.5797680616378784, -0.9366900324821472, 2.7709197998046875, 0.3706343472003937, 0.46314737200737, 0.4710329473018646, 0.8990996479988098, -0.004485597833991051, -0.7542141675949097, -0.9698781967163086, -0.5477468371391296, 1.070682406425476, 0.19489283859729767, 2.205968141555786, -2.5176689624786377, -0.1542181819677353, -0.03912670537829399, 1.1610729694366455, 1.1152639389038086, 0.8963022828102112, -0.49594762921333313, 0.16521407663822174, 0.09923470765352249, 0.9314427971839905, -1.0445468425750732, 1.2080442905426025, -0.70457923412323, 0.19391752779483795, 0.9653181433677673, 1.0491000413894653, -1.5078513622283936, -0.004578886553645134, 0.5642803907394409, -0.16017667949199677, 1.1357558965682983, -0.5226088166236877, -2.0916948318481445, 1.6055266857147217, -0.9193774461746216, 0.6350434422492981, -1.3060706853866577, 0.23367048799991608, -0.6391801238059998, -2.2643089294433594, -0.8330693244934082, -2.7357070446014404, -0.5114594101905823, -1.3468639850616455, -0.2739872932434082, -1.8134233951568604, -0.1738431304693222, -0.15909531712532043, 1.0687816143035889, 0.1271524429321289, -1.0304757356643677, 0.41533029079437256, 1.7663730382919312, 0.03052671253681183, -0.12566781044006348, 2.48002028465271, -0.31362923979759216, 0.16087177395820618, -0.37067294120788574, -0.8350704312324524, 0.2474338263273239, -0.1435857117176056, 0.1183474138379097, 0.8591694831848145, -0.11236298084259033, -0.5712912678718567, 0.7696665525436401, 0.8339431881904602, 0.097295843064785, -0.48913803696632385, -1.4558271169662476, -1.1098542213439941, 1.0213301181793213, 0.3323354125022888, -0.8392701745033264, -1.3839961290359497, 2.084737777709961, 0.6088136434555054, -1.2225708961486816, -0.19845156371593475, -1.2969573736190796, -0.1837446242570877], [-0.06610752642154694, 0.6458313465118408, -0.9872788786888123, 0.10505031049251556, 0.19316606223583221, -0.16013850271701813, -0.9300491213798523, -0.740559995174408, -1.07655930519104, 0.5550584197044373, -0.8620123863220215, 1.012937307357788, 0.5091648697853088, 1.091264247894287, 0.8427550196647644, 0.9816396832466125, 0.12339883297681808, 0.16333074867725372, 1.1304486989974976, -1.0423105955123901, 1.3222215175628662, 0.3777950406074524, 0.48500582575798035, -0.7107366919517517, 0.08542285114526749, 0.6875172853469849, -0.8049576282501221, -0.9641928672790527, 0.3145172894001007, 0.039572812616825104, -1.893410563468933, 0.08547092229127884, 0.9937247037887573, 0.5518673658370972, -0.8654128909111023, 0.9652573466300964, 0.5097197890281677, -0.2198013812303543, 0.25575271248817444, -0.5874061584472656, -0.16017229855060577, 1.7794697284698486, -0.5244411826133728, -1.4576847553253174, 0.39630764722824097, -0.3470769226551056, 0.047621045261621475, -1.3862658739089966, 0.3843323886394501, 0.6906142234802246, 0.29545238614082336, 1.8898558616638184, 0.7285355925559998, 1.7602856159210205, 0.8034546375274658, 0.3405720591545105, -1.1281238794326782, -1.0506318807601929, 0.017227482050657272, 0.6495875716209412, -0.536656379699707, 1.9562081098556519, -0.7794555425643921, -0.2943338453769684, -0.0032419951166957617, 1.4693653583526611, -0.8515229225158691, 1.0272685289382935, 1.294566035270691, 2.649894952774048, -1.2625013589859009, 1.0951247215270996, -0.5993411540985107, -0.18987181782722473, -0.4962882101535797, -1.0449366569519043, -1.721927285194397, 0.2740340828895569, -0.4940301775932312, -1.759253740310669, 0.04552210494875908, -1.4745330810546875, 1.6624419689178467, 1.5798379182815552, 0.6737737059593201, 1.046913743019104, 0.056985192000865936, 0.9367141127586365, 0.43951737880706787, -0.47431594133377075, -0.7237229347229004, -0.06830309331417084, 0.4051918089389801, -1.7264899015426636, 0.11766904592514038, -1.2171070575714111, 0.02038385719060898, 1.1570465564727783, -0.23406532406806946, -0.8107175827026367, 0.8215448260307312, 0.8603988885879517, 1.3086764812469482, -0.6965977549552917, 1.3879246711730957, 0.6828382611274719, -0.15967029333114624, -0.034792300313711166, 0.6639204025268555, -0.6379409432411194, 0.6747084856033325, -0.034352466464042664, -0.15699228644371033, -1.4595365524291992, -0.6467198133468628, -0.054355766624212265, 0.35649317502975464, -0.3072398602962494, -1.1461594104766846, 1.3881688117980957, -2.463888645172119, 0.5637539029121399, 1.897757649421692, -0.6081030368804932, 0.031349629163742065, -0.0918152779340744, -0.0844976156949997, -0.9279103875160217, -0.1531759649515152, 0.9236050844192505, -0.17447137832641602, 0.6996545791625977, 0.4616197347640991, 0.9204460978507996, 0.2612980604171753, -0.2991635501384735, -0.6579630374908447, -0.4905252456665039, 0.42896097898483276, -0.5025585293769836, 0.4786335229873657, 0.45213428139686584, 0.47948694229125977, 0.12857314944267273, 1.3166980743408203, -0.6362568736076355, -1.1945672035217285, -1.1028380393981934, -0.20159299671649933, 1.6495647430419922, 0.19842325150966644, 0.09315858036279678, 1.4684561491012573, -0.436022013425827, 2.569389581680298, 0.594194769859314, -0.45041024684906006, -0.370065301656723, -0.35202062129974365, 2.0235230922698975, -0.32461196184158325, 0.37884974479675293, -0.6760239601135254, 1.3695188760757446, 1.4256904125213623, -0.4740108549594879, 0.4717782735824585, 1.2987468242645264, 0.5063938498497009, 0.26320308446884155, -0.03977254405617714, 1.4761133193969727, -0.821529746055603, -0.09425146132707596, 0.5370416045188904, 0.7022666335105896, 0.2776950001716614, -0.49801507592201233, 0.9294033646583557, -0.45391741394996643, 0.880415678024292, 0.4358416795730591, 0.8292550444602966, 0.052791014313697815, 0.10232622176408768, -0.29630327224731445, 0.8496075868606567, -0.5978149771690369, 0.1703425794839859, -0.533552885055542, 0.5386608242988586, 1.2734169960021973, -0.4194662272930145, 0.38587555289268494, 0.24284577369689941, 0.9351006746292114, 0.5161965489387512, 0.6993825435638428, 2.504833936691284, -0.45883414149284363, 0.11784038692712784, 0.33906498551368713, -0.23394013941287994, 1.0618181228637695, 1.3757857084274292, -0.36846980452537537, -2.0268311500549316, -0.0944560244679451, 0.6508085131645203, 1.7888059616088867, 0.38929125666618347, 3.028689384460449, 0.8888049721717834, -0.9993985891342163, 1.3592129945755005, 1.1822481155395508, 0.5795087814331055, 0.6068010926246643, 0.8407430052757263, 0.8894522786140442, 0.03614833205938339, 0.808196485042572, -0.2525070011615753, 1.5170329809188843, 1.037744164466858, -0.0549745187163353, 0.3373977541923523, 0.5247738361358643, 1.381530523300171, -1.3028885126113892, 0.5382662415504456, -1.0350596904754639, 1.9577232599258423, -1.7114657163619995, -0.9146753549575806, 0.7005636692047119, -0.7566502094268799, -0.11889061331748962, -0.512884259223938, -0.2890402674674988, -0.4061349928379059, 0.5069465637207031, -0.6105353832244873, 0.596664309501648, 2.0366954803466797, -0.5946105718612671, -0.05343398079276085, -0.06727292388677597, -0.8556983470916748, -1.0448408126831055, -1.4814540147781372, 1.6006993055343628, 0.48455289006233215, 0.059734392911195755, 0.18405230343341827, -0.9781272411346436, -1.1670150756835938, 0.2086421102285385, -2.537651300430298, -0.9027047753334045, -0.5239019393920898, 0.0962354987859726, -0.8421221971511841, -0.7273014187812805, -0.09034564346075058, 0.10697366297245026, 1.5799065828323364, -0.7082970142364502, 0.7454749941825867, -2.0367748737335205, 1.2805187702178955, -0.437700480222702, 0.19446375966072083, -1.977769374847412, 1.8783966302871704, 1.600527048110962, 0.5810964703559875, -0.15942952036857605, -0.4950709342956543, 0.6526818871498108, -0.19754339754581451, -0.04394194111227989, -0.2746569812297821, 1.8352833986282349, 0.15658807754516602, 1.131142497062683, -2.539976119995117, 0.19076046347618103, 0.8606370687484741, 0.059865228831768036, -0.8054202198982239, -0.8267389535903931, 0.7634718418121338, 0.9626916646957397, -0.3372606039047241, -0.8129169344902039, 0.4426589608192444, 0.3155519664287567, -0.4099581241607666, 0.4750969111919403, -1.0201282501220703, 0.8143358826637268, -1.7494598627090454, 1.392577052116394, 0.375323623418808, 0.8340244293212891, 0.4041490852832794, 0.0765342116355896, 0.05682115629315376, 0.15218216180801392, 0.3402956426143646, -0.09344134479761124, 0.03660145401954651, 3.086773157119751, 1.3276835680007935, 0.2949310839176178, -0.7127901911735535, 0.4399645924568176, 0.1993437558412552, 0.361646443605423, 1.5964305400848389, -1.5839420557022095, 0.7665571570396423, 0.2456398606300354, -1.006767749786377, -1.85551917552948, -0.1908731609582901, 1.5722556114196777, -2.2683815956115723, 1.050935983657837, -2.0806899070739746, 0.543418824672699, 0.5199078917503357, -1.7652336359024048, 0.08991450071334839, -0.59247887134552, -1.3612308502197266, -1.0716301202774048, -0.7038550972938538, -0.8258297443389893, 0.7240613102912903, 1.8586883544921875, 0.5517939925193787, -1.1283972263336182, 0.41999003291130066, 0.22459949553012848, -1.2970775365829468, 1.2362065315246582, 0.05208756774663925, 0.6303398013114929, -0.5229780673980713, 1.6283111572265625, 0.1802431046962738, -1.071175217628479, -0.0872739851474762, 1.1304484605789185, -0.7191588878631592, -2.4608495235443115, 0.34165292978286743, 1.399194598197937, 0.01885702833533287, -0.8834207057952881, 0.38108542561531067, 1.1149197816848755, 0.17551831901073456, -2.3904216289520264, 0.7212840914726257, 0.7959141135215759, 0.10772954672574997, -0.689080536365509, -0.1997254490852356, -0.12107741832733154, 0.5137356519699097, -0.9007426500320435, 1.0326266288757324, -0.7697905898094177, -1.3660578727722168, -1.0150985717773438, -1.0378344058990479, 0.02285071276128292, 1.2528762817382812, 1.8069167137145996, -0.15908695757389069, -0.8602714538574219, -1.6401629447937012, 0.4175839126110077, 1.1836930513381958, -1.6232794523239136, -1.6330747604370117, -0.8996224999427795, -0.07035215198993683, -0.04353497549891472, 1.0840297937393188, -0.4343705177307129, 0.07485029846429825, -1.9515999555587769, -0.525286853313446, 0.6495180130004883, -0.49606844782829285, 1.092393398284912, -0.09662124514579773, -0.8479723930358887, -0.1409585326910019, 1.5829880237579346, 1.8035666942596436, -0.11861102283000946, 1.7043691873550415, -0.45621296763420105, -0.5174163579940796, 2.165801525115967, 0.7627226710319519, 0.39411643147468567, 0.9739342927932739, -0.24166101217269897, 0.3364489674568176, -2.4833414554595947, -1.388364315032959, 0.8895226120948792, -0.6275866031646729, -0.7484313249588013, -1.0812331438064575, -0.019902192056179047, -0.6981483697891235, -0.028395051136612892, -2.524570941925049, -0.7252213954925537, 0.06659345328807831, -1.5729550123214722, 0.960372269153595, 1.2006288766860962, 0.8380616903305054, 0.8602918386459351, 1.5098652839660645, 0.24948233366012573, 0.11852467060089111, -0.9321494698524475, -0.8510677218437195, 0.8273946642875671, -0.6493855118751526, 1.5802533626556396, 0.9153878092765808, 1.298061490058899, 0.33371251821517944, -1.5089982748031616, -0.18447209894657135, 1.8504281044006348, 0.2966936528682709, 0.322268545627594, -0.9897550344467163, 0.02132141776382923, 1.4825985431671143, 0.18184977769851685, -0.44654056429862976, -0.7098824977874756, 0.8990968465805054, 0.9491571187973022, 0.9787746071815491, 0.5696782469749451, -1.3380694389343262, -0.6703961491584778, 1.142740249633789, -0.7041663527488708, -0.8378856182098389, 0.09529193490743637, -0.2040059119462967, -0.36913833022117615, -0.43739306926727295, -0.8782234191894531, -0.2772180140018463, -1.5292398929595947, 0.22018519043922424, -1.1368318796157837, -0.45634037256240845, -0.06566338986158371, -0.40614157915115356, 0.7770353555679321, -0.5030527710914612, 1.7870343923568726, -1.305857539176941, -1.4098756313323975, 0.9958122372627258, -0.17766688764095306, 0.3628619909286499, 2.0549983978271484, 0.9244686961174011, -1.828535556793213, -1.4899580478668213, 0.49154117703437805, 1.6113920211791992, 1.0863559246063232, -0.702656090259552, -0.46525076031684875, -1.304793357849121, -0.8933441042900085, -0.5521937608718872, 0.005200504790991545, -0.4470767080783844, -0.2867509722709656, 0.2834434509277344, -0.6560792922973633, 0.7265899181365967, 0.6417099237442017, 0.6894785165786743, -0.6012441515922546, 0.36152157187461853, 0.5701304078102112, -1.2454124689102173, 0.8535985350608826, 0.11678752303123474, -0.20888125896453857, 0.7662831544876099, 1.0336397886276245, 0.35661840438842773, -1.2650614976882935, -0.9799116849899292, 0.8572816252708435, -0.2615705728530884, 0.7175558805465698, 0.9196908473968506, -0.3225117027759552, 0.2651512324810028, 0.7299210429191589, -0.198443204164505, 0.624598503112793, -1.0421463251113892, -0.37925347685813904, -0.13405202329158783, 0.6734089851379395, 0.3894824981689453, -1.0892876386642456, 2.2842142581939697, 1.4342725276947021, 2.7838680744171143, 2.4320130348205566, -1.2755831480026245, -0.010538297705352306, -0.5379998683929443, 0.2892768383026123, 1.650882363319397, 0.7270505428314209, -0.5701608657836914, 2.198700189590454, 0.3227290213108063, 1.1504323482513428, -0.9677093029022217, 0.45758697390556335, -2.0761098861694336, -0.28996166586875916, -1.3835580348968506, -0.49801895022392273, 1.761977195739746, -1.008753776550293, 0.3691413104534149, 0.39301714301109314, 0.24432030320167542, -0.385847270488739, -0.7812361121177673, 0.9679756164550781, -1.1249974966049194, -0.71514493227005, -1.1315380334854126, -1.5660293102264404, -0.8358011841773987, -0.7120617628097534, -0.08065805584192276, 1.3697099685668945, -0.633708655834198, 0.496715784072876, 0.2054978907108307, 0.5800427794456482, 0.42541906237602234, -0.6369233727455139, -1.1402837038040161, 0.6161660552024841, 0.6173481345176697, -2.32936954498291, 1.1444964408874512, 2.0506176948547363, -0.8977370858192444, 1.903315782546997, 1.9245672225952148, -0.639320969581604, 1.8774601221084595, 0.2572946548461914, 0.13131771981716156, -0.6344196200370789, 0.04474666714668274, -0.20574334263801575, 1.1411285400390625, 1.2620141506195068, -0.5658401846885681, -0.4500633180141449, 0.21919018030166626, 1.9871097803115845, 1.7175225019454956, 0.8404738903045654, -2.8324742317199707, -1.1432466506958008, 1.4807827472686768, -0.9313231706619263, 1.3319770097732544, -0.12773732841014862, -1.215394139289856, 0.34853172302246094, 0.9933860898017883, 0.11587688326835632, -1.2598391771316528, -0.7605480551719666, 1.3028786182403564, 0.6667441129684448, -0.6998975872993469, 0.3123452067375183, 0.2996605336666107, 2.0244181156158447, 0.307609885931015, 1.3577072620391846, -0.6476963758468628, 0.5476566553115845, -0.6221511363983154, 1.7262376546859741, 0.9628309011459351, 0.921545147895813, -1.0209380388259888, 0.1646919548511505, 0.8526633381843567, -2.192460775375366, -0.17456749081611633, 0.5364715456962585, 0.05471469461917877, -0.5091131329536438, -0.7625616788864136, -3.012451648712158, -0.6092548966407776, -1.1225227117538452, 0.4913434386253357, -0.314189612865448, -1.0767273902893066, 0.6460214257240295, -0.8178871273994446, -0.28216928243637085, -0.13660842180252075, 1.3886089324951172, -2.2039270401000977, -1.4517393112182617, -1.6193703413009644, 1.4178115129470825, 0.14119866490364075, 0.7499200105667114, 0.8104521632194519, -0.3668903410434723, -1.0671309232711792, -1.709010124206543, -0.19828639924526215, 1.0782188177108765, 0.27669864892959595, -0.44932445883750916, -0.776847779750824, -0.2512015700340271, 0.5042701363563538, 1.3078501224517822, 0.43596217036247253, 1.3704475164413452, 0.37486138939857483, -0.37728151679039, -0.20871245861053467, -2.3988277912139893, 0.7385309934616089, 1.7810981273651123, -0.06679143011569977, -0.5458629131317139, -0.18212564289569855, 1.6137521266937256, -0.7707635760307312, 0.9589942097663879, -2.332709789276123, 0.9761461019515991, -0.9218939542770386, 2.765423059463501, 1.000420331954956, -1.1600370407104492, 0.9296514391899109, -0.8152337670326233, -0.08455123007297516, 0.9146497249603271, 2.4264771938323975, -0.6352558135986328, 0.01865422911942005, -0.07858029007911682, 1.1395467519760132, -0.8356735706329346, 0.1351403295993805, 2.160348892211914, 1.4963728189468384, 0.23052357137203217, -0.45984533429145813, 1.3067466020584106, -1.3121815919876099, 1.7837668657302856, -0.06289132684469223, -0.6284560561180115, -0.12354054301977158, -0.7945151329040527, -0.9011020064353943, -1.411726474761963, 0.021050162613391876, 1.4199963808059692, 1.226303219795227, 0.18100135028362274, -0.14561252295970917, 0.9116799831390381, -0.43710777163505554, 0.03870111331343651, -1.1627260446548462, -0.6572806239128113, 1.5402504205703735, -1.6164135932922363, -1.9793864488601685, 0.6905134916305542, 1.153411865234375, 0.4574960768222809, -0.23187893629074097, -0.8386223912239075, 0.2867266833782196, -1.09666907787323, 0.6200653314590454, -0.9012062549591064, -2.3286616802215576, 0.6070429086685181, 0.19156508147716522, 1.3096784353256226, 0.6527567505836487, 1.9279088973999023, -0.28209275007247925, -0.008893899619579315, 1.213189959526062, -0.05713194981217384, -0.09727199375629425, -0.3536740839481354, 1.4794267416000366, 0.20698492228984833, -0.856808066368103, 1.9749311208724976, -0.2488132119178772, 2.287654161453247, -0.6843579411506653, 0.29484230279922485, 2.646005868911743, -0.8241861462593079, 0.3348560631275177, -0.7405791282653809, -1.2420432567596436, 1.8024426698684692, -0.5128978490829468, 1.873689889907837, 0.7047446370124817, -0.7093431353569031, 0.8582590222358704, 0.04911482334136963, -0.16975371539592743, 0.19427916407585144, -0.6737256646156311, 0.2134947031736374, 0.25842612981796265, 1.3595919609069824, -0.673207700252533, 1.2833367586135864, 0.30309969186782837, 0.3027714788913727, 1.6924697160720825, -0.04892624914646149, 0.16545231640338898, 1.1267322301864624, 1.367483139038086, -0.23984834551811218, 0.4210112988948822, -0.5366824269294739, -0.9953250885009766, -0.8777428269386292, 1.7434427738189697, 1.0087321996688843, -0.1272256076335907, 0.6664169430732727, -2.059516429901123, -0.8806849718093872, 0.23997743427753448, -0.5079036951065063, -0.5828758478164673, 1.508101463317871, 0.715456485748291, -1.235119104385376, -0.6371166110038757, 0.06644444167613983, -1.4603909254074097, 0.051924675703048706, 0.49935853481292725, 0.2515125572681427, 0.8654443621635437, 2.2144670486450195, 0.7053348422050476, 0.36631208658218384, 0.327452689409256, -0.4719395339488983, -0.5388933420181274, -0.7202200293540955, 1.7018343210220337, -1.1827421188354492, -1.0220732688903809, 0.7090767025947571, -0.8929274082183838, -1.0234404802322388, 1.5723276138305664, -0.44890767335891724, 0.8583366870880127, 1.311819314956665, 0.6909574866294861, -0.7005791664123535, 1.4214633703231812, 0.029632018879055977, -0.2623363733291626, 0.2997773587703705, 0.4016698896884918, 0.011952788569033146, -1.2184150218963623, 1.1901884078979492, 0.2345622181892395, 0.7838843464851379, 0.9335025548934937, 0.5457257032394409, -0.870619535446167, -0.9570199847221375, -0.5902444124221802, -1.0561426877975464, 0.6404225826263428, 0.08280255645513535, -0.4175903797149658, -0.23799218237400055, -1.3929437398910522, 1.3803722858428955, 0.38403210043907166, -1.198770523071289, -2.6081900596618652, -0.2988455295562744, 1.7118818759918213, -1.9367550611495972, 0.13110758364200592, -1.4836513996124268, -0.6287534832954407, -0.09766066074371338, -0.7698332071304321, 0.6648926138877869, -0.28000715374946594, 1.5154083967208862, -0.2388499528169632, -0.06413634121417999, -0.30850720405578613, -0.40722694993019104, -0.6521146893501282, 0.6062572598457336, -1.3479409217834473, 1.2399160861968994, -0.8823595643043518, 0.6194551587104797, -0.6287261247634888, 0.5058301687240601, -1.7266193628311157, -0.08527851104736328, -0.8515196442604065, 1.0832371711730957, -2.4981555938720703, -0.7266843318939209, 0.7814324498176575, 1.3760106563568115, -0.3794546127319336, 0.186813086271286, 2.5057196617126465, -1.7111455202102661, 0.4996410310268402, 1.2947248220443726, 0.3396006226539612, -0.07438930124044418, 0.9534810781478882, 0.05997570976614952, -1.3069231510162354, 0.9161542654037476, 0.46073490381240845, 0.67657071352005, -2.2998266220092773, 1.2264275550842285, -0.2697252333164215, 0.0806725025177002, 1.1237857341766357, -0.16441647708415985, -0.9439518451690674, -1.215340256690979, -0.20708930492401123, -0.7096427083015442, 1.523617148399353, -2.1987316608428955, 0.8706779479980469, -0.6224371790885925, 0.4268629550933838, -0.7647576928138733, 2.6138412952423096, 0.14266768097877502, 0.1724400818347931, 0.23304930329322815, -0.9472002983093262, 0.2920120358467102, -0.13056834042072296, -0.6244187951087952, -2.136507749557495, -0.8982012867927551, -0.4996999204158783, -1.3185960054397583, -0.4769419729709625, 0.9717460870742798, 0.5896172523498535, 0.12368054687976837, 0.6605395674705505, -0.7313449382781982, -0.011496366932988167, 0.0707106813788414, -0.5208067297935486, -0.2799680233001709, 0.5254969596862793, -0.6563806533813477, 0.17834748327732086, -0.3923478424549103, 0.1000334843993187, -1.0445843935012817, 0.17223821580410004, -0.4287998080253601, 1.761596918106079, -1.3743739128112793, 0.2799634635448456, -0.0647948831319809, 0.4097493588924408, -0.4015932083129883, 2.24137282371521, -1.4490565061569214, -1.8117876052856445, -1.4515560865402222, -1.3156598806381226, 0.1954270601272583, -0.08287464827299118, 0.17463421821594238, -0.5524882674217224, 0.028502797707915306, 1.6464167833328247, -0.4578510820865631, 0.027503959834575653, 1.023359775543213, -0.15594804286956787, -0.8461784720420837, -1.4275330305099487, -0.8879306316375732, 0.7624205946922302, 0.7419717311859131, 0.5818057060241699, -0.19220735132694244, 0.5755335688591003, 1.3441895246505737, -0.6428776979446411, 1.1041450500488281, -0.3602118194103241, 0.10980162769556046, 0.08031009137630463, 2.104180335998535, -1.41396164894104, -0.45808541774749756, -0.048697467893362045, -0.17499370872974396, -0.07929020375013351, 1.5398240089416504, 1.3761415481567383, 0.5057907104492188, -0.1655360907316208, -2.0890965461730957, -0.8164809346199036, -0.9380837678909302, 0.7488057017326355, 0.472922146320343, 1.1294269561767578, 0.5810292363166809, -0.20891140401363373, 1.1767123937606812, -0.538707971572876, 1.9300475120544434, -1.952985167503357, -0.09520460665225983, 1.746300458908081, -0.030423378571867943, 0.25598055124282837, 1.2237766981124878, -0.5131767988204956, 1.2389335632324219, 1.264207363128662, -0.9893686175346375, 0.4117911458015442, -0.7288879156112671, -1.1337568759918213, -0.49274566769599915, 0.9667905569076538, 1.7158851623535156, 1.6181601285934448, 0.5171089172363281, 0.23010727763175964, 0.07415361702442169, -0.5790202021598816, 0.3508986830711365, -0.09189387410879135, 0.013369173742830753, 0.661373496055603, 0.6723456382751465, -0.6503050327301025, 0.38410770893096924, -0.4180530905723572, 0.11407171189785004, 0.5501971244812012, 0.4569489657878876, -0.8284391164779663, -0.7090038657188416, -0.6195148825645447, -0.8454283475875854], [2.4302334785461426, 0.08599597215652466, 0.6507744193077087, -0.1394868642091751, 1.0053225755691528, -0.20557217299938202, 2.3107903003692627, 0.9611963629722595, -2.4588804244995117, 1.096246600151062, -0.0649130567908287, 1.087007999420166, 1.4803025722503662, -0.32822170853614807, -1.8499090671539307, 1.4050508737564087, -1.2190543413162231, -0.3078932464122772, -0.8239351511001587, 1.1244124174118042, 0.9127874970436096, 1.4789375066757202, -1.115782380104065, 0.01862475648522377, 1.0811116695404053, -1.0803368091583252, -0.13723339140415192, 0.22912432253360748, -1.225288987159729, 0.5667101740837097, -1.0115634202957153, -0.6942768096923828, 0.05807768553495407, -1.5520224571228027, 0.2389381378889084, 0.2635759115219116, 0.4817185401916504, 1.0240761041641235, -0.1040261760354042, -0.1287836730480194, 0.41494739055633545, -0.11001957207918167, -0.9257010817527771, -0.10334918648004532, -0.730495035648346, -0.3965704143047333, -0.30169180035591125, 1.103991150856018, 0.16146431863307953, -0.006268360652029514, -0.4664003551006317, 0.2772439420223236, -0.4961291253566742, 1.5266742706298828, 0.050046999007463455, 0.8767049312591553, -0.8596596717834473, -2.4731154441833496, 1.4208354949951172, 1.5798122882843018, -1.7454313039779663, -0.714270293712616, -0.2320656180381775, -0.19154901802539825, -1.4888073205947876, 0.04092295840382576, -0.9305246472358704, -1.9530878067016602, -0.004458647686988115, 0.6479201316833496, -0.7969794273376465, -0.6382805705070496, -0.3327442705631256, -0.8369497656822205, -0.3398703336715698, 0.7555641531944275, 1.8263399600982666, 0.9387630224227905, 0.4393489956855774, -1.410603642463684, 0.6936215162277222, -0.08230932056903839, -0.4817522168159485, -0.34986168146133423, -0.8249951601028442, 0.8901103138923645, -0.7080342769622803, 0.8302658796310425, 0.5278708934783936, 0.5453546047210693, 1.0000265836715698, 0.3290864825248718, -0.780825674533844, -0.2243523746728897, 0.20067544281482697, -0.10826349258422852, -1.1716099977493286, -0.705463707447052, 0.48217734694480896, -0.44580012559890747, -0.5260231494903564, -2.232133626937866, 1.0399917364120483, -0.23734143376350403, 1.151069164276123, 0.4151493310928345, 0.44745054841041565, -0.5217363834381104, 1.6303637027740479, -0.45252859592437744, 0.9186338782310486, -0.30552858114242554, 1.0433214902877808, 0.8546016812324524, -0.406658798456192, -0.26480454206466675, 1.8675936460494995, 0.7711300849914551, 0.7558439373970032, 1.1903924942016602, 0.8009100556373596, -0.5669965147972107, 0.1977628618478775, 0.42338353395462036, -0.5224677920341492, 2.131287097930908, 0.1407202035188675, 0.6913431286811829, 0.6737555861473083, -0.8484101891517639, -0.8093745708465576, -0.3783781826496124, 0.6048209071159363, -0.7730095386505127, 1.9787155389785767, -1.2539242506027222, -0.17747566103935242, -0.5055856704711914, -0.09911986440420151, -0.3648166358470917, -0.9824313521385193, -1.294600009918213, 0.89876788854599, 0.5464710593223572, 0.4972188174724579, 1.646467685699463, 0.17945852875709534, 0.626625657081604, 0.6658682227134705, -0.007060985546559095, -0.20527684688568115, 0.33859118819236755, -0.4998154044151306, 0.7938723564147949, -0.9191008806228638, -1.148675799369812, -1.547549843788147, -0.29734158515930176, 0.12829844653606415, 0.400745689868927, 1.457018256187439, -0.271683007478714, -0.0933598056435585, -0.4715254306793213, -0.12141850590705872, -1.1755759716033936, 0.2763713598251343, 0.547544538974762, 0.864422082901001, -0.29871872067451477, -0.5420187711715698, -0.1678207814693451, -0.034546177834272385, 0.5837258696556091, -0.12506188452243805, 0.14698165655136108, 0.8828083872795105, -0.667351484298706, 0.16258133947849274, -1.430217981338501, -0.8312138915061951, -1.1191673278808594, -0.5327950119972229, 0.8170785903930664, 2.1394901275634766, -0.9463821053504944, -1.9249967336654663, -0.907802164554596, 0.2056547850370407, -1.5728743076324463, 0.2863578200340271, -1.0350568294525146, 1.1489120721817017, -0.9551282525062561, 0.5634664297103882, -1.0578339099884033, -0.3272349536418915, 1.1257565021514893, -1.0615081787109375, -0.003177579725161195, -2.063236713409424, -0.15029025077819824, -0.7039585709571838, -0.12245263159275055, 0.9720492959022522, 1.0606627464294434, -0.36735132336616516, -0.2924943268299103, -2.0635464191436768, -2.592074155807495, 0.5287295579910278, 1.4458458423614502, -1.0542633533477783, 0.048641182482242584, 0.5646811723709106, -0.8700498342514038, -1.0879113674163818, 0.2869102358818054, 0.6483840346336365, -2.007929801940918, -0.1953384429216385, 1.261691927909851, 0.12669451534748077, 0.3198827803134918, 1.6513347625732422, 1.3145946264266968, 0.6555954217910767, -0.3711807131767273, 0.4453670382499695, 0.350894033908844, -1.3262979984283447, -0.7594103813171387, 1.3251320123672485, 0.8427306413650513, -2.369785785675049, -0.05366756394505501, -1.0001403093338013, 0.5179911255836487, -0.4145342707633972, 0.18134170770645142, -0.286649614572525, -0.4850054681301117, 1.1525925397872925, 1.1567026376724243, 0.5690934658050537, -0.5559636950492859, 0.21453982591629028, -1.542101263999939, -0.907474935054779, 0.30091753602027893, -0.606681764125824, 0.15283453464508057, -0.18186801671981812, -0.2559308707714081, -0.09404341876506805, 0.4606493413448334, 0.5018075108528137, -1.897464632987976, 1.1842330694198608, -2.244320869445801, 0.9239659905433655, -1.0235008001327515, -0.3672410249710083, -0.9990833401679993, 0.8033390641212463, -0.5902813076972961, 1.1700056791305542, 0.2010568380355835, 0.15044434368610382, 0.5515211820602417, 0.780662477016449, -0.13612599670886993, -0.240947425365448, -0.42323943972587585, -1.6732763051986694, 0.3351762294769287, -0.14821626245975494, -1.5992776155471802, -0.28100836277008057, 0.5604789853096008, -1.0582680702209473, 0.07423906028270721, 0.29508182406425476, -0.17036616802215576, 1.3515914678573608, -0.42393532395362854, -1.0329036712646484, -0.9941266775131226, 0.5604333877563477, -0.9533037543296814, 0.7231680750846863, 1.0094683170318604, 0.20264849066734314, 0.7154605388641357, -2.0286502838134766, -1.1714997291564941, -0.6291415691375732, 0.9286593198776245, 0.38743433356285095, 0.26884979009628296, -0.3034992814064026, -1.076890468597412, 0.7928074598312378, 0.14700005948543549, 0.6559651494026184, 0.341885507106781, 0.505301296710968, 0.4127596616744995, 1.6719260215759277, -0.5749570727348328, -0.8652191758155823, 0.20564717054367065, 2.9438562393188477, 0.5370520949363708, -0.5913640260696411, 0.5184768438339233, -0.23705005645751953, 0.3444964587688446, 0.007478555664420128, -0.4478023648262024, -0.05101566016674042, 0.6216654181480408, -1.143942952156067, 1.1883496046066284, 0.1335185170173645, 1.6447778940200806, 0.7718300223350525, 1.5402780771255493, -2.349287748336792, -0.8058477640151978, -0.8895283937454224, 1.1201280355453491, -0.8799810409545898, -0.575617253780365, 1.128935694694519, -0.4912290573120117, -1.3693468570709229, -1.1065276861190796, 0.5599987506866455, 0.27128222584724426, -0.503516674041748, -0.3396652340888977, -1.2961742877960205, -0.5357909202575684, 0.5795047879219055, -1.1929783821105957, -0.4425605237483978, 0.8919708728790283, 0.6763021945953369, 0.2206469625234604, 0.5575093626976013, 1.9416968822479248, 1.21001136302948, 0.30340176820755005, 0.8958607912063599, -0.07642944157123566, 0.6184973120689392, 1.1664842367172241, 2.36785888671875, -0.502108633518219, -1.086972951889038, -0.09834557771682739, -1.9471421241760254, -0.8185177445411682, 1.5118253231048584, 0.21281179785728455, 1.7619131803512573, 0.5214284658432007, 1.0128991603851318, -1.4717069864273071, -1.2514945268630981, 2.0983009338378906, 0.3747681677341461, -0.5873768329620361, 0.23627735674381256, -0.8672164678573608, -0.4946190416812897, -0.5663015246391296, 0.9815540909767151, -0.40344110131263733, -1.7272632122039795, -0.4279891550540924, -0.35914990305900574, 1.3180135488510132, -0.18413470685482025, 0.1990295648574829, -1.7144333124160767, -2.014760971069336, 1.0481191873550415, -0.9763248562812805, 1.182460904121399, 0.27566906809806824, 0.5100727677345276, 2.751305103302002, 0.27351921796798706, 0.4862914979457855, 0.46192020177841187, -1.4825245141983032, 0.5971917510032654, 0.8234589099884033, -1.8366371393203735, -0.4624636769294739, -1.1397507190704346, -0.39263731241226196, 1.8687126636505127, -0.043338216841220856, 0.4024626612663269, 0.21128138899803162, -0.1291959136724472, -0.6315187215805054, -0.039778608828783035, 0.07664734125137329, -1.2779793739318848, 0.44998231530189514, -0.3115227520465851, 0.5446264147758484, -1.0781499147415161, 0.351563423871994, 1.6094521284103394, 2.2427639961242676, 0.13711024820804596, -1.276989459991455, 0.7476682662963867, -0.43338167667388916, -1.2444052696228027, 0.17636547982692719, 1.084739327430725, -0.06152406707406044, 0.47220414876937866, -0.8973031044006348, 0.16414396464824677, -0.5225152373313904, 0.42513397336006165, 1.500383973121643, 0.20746836066246033, 1.0850269794464111, -0.5436709523200989, 1.6424474716186523, 0.6508474349975586, -0.3179585337638855, -0.5789024829864502, -0.5647817850112915, -0.7529733180999756, -0.2991481423377991, -0.3386951684951782, -0.8337012529373169, -0.3410342037677765, 1.6408360004425049, -0.24168890714645386, 0.45832595229148865, 0.9260208606719971, -1.023979902267456, 0.29421013593673706, 0.5516437888145447, 0.6526342034339905, 0.8065317273139954, 0.1668703258037567, 1.0009530782699585, 0.23400147259235382, 0.959225058555603, -0.37055420875549316, -0.9312009811401367, -0.325780987739563, -0.3689597249031067, -0.6737709641456604, 0.3806503713130951, 1.2791088819503784, 0.9643725156784058, 0.8003936409950256, 0.8846951127052307, 0.8678702712059021, 1.1324365139007568, -0.578091561794281, -0.8934411406517029, -0.842064619064331, -0.6087905168533325, -0.7475113868713379, -0.17252318561077118, 0.9742904901504517, 1.3634089231491089, 0.2112058401107788, -0.3378452956676483, -0.759772002696991, 0.10734689980745316, -0.2532840371131897, 0.595829963684082, -0.23772743344306946, -1.2541240453720093, -1.4788498878479004, -1.0289608240127563, 0.5674223303794861, -1.6692129373550415, 1.0069018602371216, -0.3154539167881012, 0.15201140940189362, 0.8628444075584412, -0.48314017057418823, 0.07692191749811172, 0.2083667367696762, -2.0151548385620117, 1.739795207977295, -0.6132974028587341, 1.3343071937561035, -0.8553280234336853, -1.1930265426635742, -0.4074951410293579, 0.9332932829856873, -0.23602525889873505, 0.5533186197280884, 0.5628345012664795, -1.9181199073791504, -0.021575259044766426, 0.11220355331897736, -1.3949581384658813, 0.18528926372528076, -0.8236274123191833, 0.3728886842727661, 0.5074455738067627, 0.04177434369921684, 1.1361446380615234, -0.06308319419622421, 1.3115770816802979, 1.6353867053985596, -0.17526444792747498, -0.07571374624967575, -0.01852259784936905, -0.18243061006069183, 1.1787539720535278, -0.1487918347120285, 0.10661881417036057, -0.9977353811264038, 0.7770490050315857, 0.04227754473686218, -1.2046818733215332, -0.31200993061065674, -1.1637576818466187, 0.610639214515686, 1.291972041130066, -0.772217333316803, -0.8318370580673218, -0.09983833879232407, 0.8710411787033081, -0.49807465076446533, -0.3871331810951233, -0.6629070043563843, 1.0747997760772705, -0.17832829058170319, -0.20735760033130646, 0.1943477839231491, -0.6933817863464355, -1.1597990989685059, -0.592033863067627, -2.3654496669769287, 0.10902386158704758, 0.37105435132980347, 1.1221100091934204, -0.4245949685573578, -0.5174515843391418, 0.6796857118606567, 1.2415053844451904, 0.537725567817688, -0.552692174911499, 0.3304829001426697, -0.08154729008674622, -0.6310194134712219, 0.11523868143558502, -1.1020276546478271, 0.07234986871480942, 1.3108611106872559, 0.9008740186691284, 0.25678345561027527, 0.3397998809814453, 0.15141473710536957, -1.3703314065933228, 0.20304842293262482, 1.071407437324524, -0.7875127792358398, 0.7427792549133301, -1.7207105159759521, 0.4018781781196594, 1.5241694450378418, 0.20444828271865845, 0.330291211605072, -1.2201087474822998, 1.1006218194961548, -1.479382872581482, 2.109462022781372, 0.5336910486221313, 0.22422932088375092, -0.9155809283256531, -0.800441324710846, 2.2982840538024902, 0.40493708848953247, -0.7682432532310486, 0.6627475023269653, -0.5009768009185791, -1.0742552280426025, -0.13219597935676575, -0.15130871534347534, -0.31717392802238464, -0.7916589379310608, -0.6947433948516846, 0.37511104345321655, 0.2811910808086395, 0.25413480401039124, -1.3090925216674805, -1.6766330003738403, -0.42845895886421204, -0.23862197995185852, 0.33890435099601746, 0.7232231497764587, 0.15554064512252808, 0.7054620981216431, -0.707753598690033, 0.5243070125579834, -1.1859490871429443, -0.7573257684707642, -0.4425140917301178, 0.9907059073448181, 1.3535212278366089, -0.4842752516269684, -1.1334611177444458, 1.2585134506225586, 0.038769472390413284, -1.4864705801010132, -0.028767669573426247, -1.6032806634902954, 0.08176034688949585, 0.5811790823936462, 0.7989587187767029, 0.08489891141653061, 0.32239025831222534, -0.9919453859329224, 0.38593918085098267, -1.8908718824386597, -0.765601634979248, -0.13039343059062958, -1.248090386390686, 0.8661556839942932, -0.0017056219512596726, -0.2010682225227356, -0.03982928767800331, -0.16066308319568634, -0.904680609703064, -0.30275869369506836, -2.2636125087738037, 0.3067190945148468, 0.1510009765625, -0.9947189688682556, -1.3744935989379883, -0.038263652473688126, -0.408769428730011, -0.11371169239282608, 0.9925329685211182, -0.980541467666626, 0.8893696665763855, -1.7576735019683838, 1.5727440118789673, 0.741403341293335, -0.5501236915588379, -0.46174654364585876, 2.0890707969665527, -1.115399956703186, 0.2674938142299652, -0.941084086894989, 0.44199368357658386, -1.8533962965011597, -0.27117398381233215, 0.6730861663818359, -0.6669543385505676, -0.25499966740608215, 0.6233699321746826, 0.5123336911201477, -0.4136105477809906, -0.7037569284439087, -2.1046783924102783, 0.012666719034314156, -0.1641993224620819, 0.9858439564704895, -1.2931331396102905, -1.8424060344696045, -1.2832868099212646, 0.41414403915405273, -0.6152921319007874, 0.9533618092536926, 0.7174065709114075, 1.8154089450836182, 0.03671552985906601, -0.460277795791626, -0.8926035165786743, 0.09143613278865814, 0.388214111328125, -0.7658644914627075, -0.07193444669246674, -0.1595086306333542, 0.38577449321746826, 0.8980641961097717, -0.2071901559829712, -0.2936439514160156, 1.045615315437317, 0.4301761984825134, 0.7879401445388794, -0.2503313720226288, 0.33368992805480957, 1.3598401546478271, 0.16739141941070557, -0.3725951015949249, -0.7856696248054504, -0.35474851727485657, 0.9301100969314575, 0.37501245737075806, 0.4569756090641022, 0.0375150702893734, 1.274668574333191, -1.3593950271606445, -0.648120641708374, 0.5740023255348206, -0.1076667457818985, 0.06808355450630188, -0.8940564393997192, -1.4529635906219482, 0.821748673915863, 1.0488359928131104, 3.639801263809204, -0.6900429725646973, -1.3823668956756592, 1.0936872959136963, 1.8980567455291748, 0.9379084706306458, 0.8612095713615417, 0.23772472143173218, 1.137840986251831, 0.6188681125640869, -0.0655788779258728, -1.1457188129425049, 0.48796316981315613, 1.3749151229858398, 1.5451207160949707, -1.7899366617202759, 0.9294756650924683, -0.9410083293914795, 1.1707262992858887, 0.32813209295272827, -1.484122395515442, 0.26520997285842896, 0.06252529472112656, 1.402682900428772, 0.2220575362443924, 0.9729520082473755, 0.7783471941947937, -0.15655723214149475, -0.21977566182613373, -0.6444348096847534, -0.07406457513570786, -0.788655161857605, 1.0665631294250488, 1.3010514974594116, 0.10247667878866196, -1.3930796384811401, -0.0070977709256112576, -0.2924138903617859, -0.656617283821106, -0.4067741334438324, 0.08215348422527313, 0.9871051907539368, -0.12144798785448074, 1.3129048347473145, -0.9495452642440796, -0.2423446774482727, 2.8754689693450928, 0.12068062275648117, 1.1803996562957764, 1.029906988143921, -0.615229070186615, 0.48184409737586975, -0.8612881302833557, -0.5651347637176514, -0.7562433481216431, -0.30603837966918945, 0.9697136282920837, 1.3982683420181274, 0.005090481135994196, -0.8391326665878296, 1.1661499738693237, 0.9484879374504089, -0.13050445914268494, 0.8464123606681824, -1.0282659530639648, -0.7480887770652771, 0.7965328097343445, 0.9061086177825928, 0.05504005774855614, -0.5233383774757385, 0.37568822503089905, 0.1118977963924408, 1.5542258024215698, -0.7278175354003906, -2.3180088996887207, 0.6880437731742859, -0.33024731278419495, 0.08432500809431076, 0.4981563985347748, 0.09451042860746384, -0.3737034797668457, -0.5254619121551514, 0.43372875452041626, 0.5335864424705505, -0.045654457062482834, 0.520666241645813, 1.280922770500183, 2.1707441806793213, -1.9609111547470093, -0.6595553755760193, 0.5857479572296143, -1.5032968521118164, 0.2067779153585434, -0.5238821506500244, -0.6284249424934387, 0.34080594778060913, 0.9672147035598755, -0.6628497242927551, 1.033313274383545, -0.26690608263015747, 0.6554157137870789, 1.037658929824829, 0.3836687505245209, 0.7543864846229553, -0.6678152084350586, 1.9808520078659058, -0.11914937943220139, -1.9336689710617065, -0.967965841293335, 0.7389468550682068, -0.34355756640434265, -0.2594298720359802, 0.7620439529418945, 0.2146180272102356, -0.5571321249008179, -0.5608828067779541, 1.477333426475525, -1.2463657855987549, 1.4641577005386353, -0.6211098432540894, -0.6505429148674011, -0.41616466641426086, 0.8591769933700562, 1.9827494621276855, 0.5469682216644287, -0.6724787950515747, 0.1642548143863678, -0.8414602875709534, -0.545598030090332, -1.60945463180542, -0.24224866926670074, -1.1225049495697021, 0.08615756034851074, 1.408961296081543, -2.0452845096588135, 0.31253039836883545, -2.140902280807495, 1.351681113243103, -0.3509957790374756, -1.1295288801193237, -0.3486110270023346, -0.8444826602935791, -0.25655558705329895, -0.6327719688415527, 1.6376138925552368, 0.23858986794948578, 0.7198386192321777, 0.034952424466609955, 0.06075466796755791, 0.6312718391418457, 0.36620888113975525, -0.3559281826019287, 0.957854688167572, 0.5814958214759827, 0.6779917478561401, 0.8525335788726807, 0.35490167140960693, -0.05133911222219467, 0.3673504889011383, -0.262636661529541, -1.5720080137252808, 1.0391826629638672, -1.077457308769226, 1.4394710063934326, -0.4678971767425537, -1.1919376850128174, -0.5229449272155762, -0.15359997749328613, 0.13186582922935486, -1.5073739290237427, 0.023634357377886772, -0.5191576480865479, 0.7664321064949036, -0.011639812961220741, -0.1701326072216034, -1.146720051765442, -0.9892778992652893, -2.0443055629730225, -0.20448315143585205, 0.20866762101650238, 1.2750109434127808, -0.21281445026397705, 0.6429311633110046, -0.6023661494255066, 0.7799667119979858, -2.346679449081421, 0.8962219953536987, -0.5411818623542786, -0.251463383436203, 0.45619598031044006, -0.7702904939651489, 0.35494595766067505, -0.06415913254022598, -1.2328537702560425, 0.4121318459510803, 0.19588738679885864, -0.5667593479156494, -0.9006873369216919, -0.5326799750328064, 0.20269790291786194, 0.17927445471286774, 1.1277742385864258, -0.4706745445728302, 0.7982763648033142, 0.46349281072616577, 0.0771767720580101, 1.5506854057312012, -0.8986750245094299, 1.7206692695617676, -0.6544780135154724, 0.4040573835372925, -0.6545436978340149, -1.8832838535308838, -0.6924386620521545, -0.35831764340400696, 0.7008156180381775, 0.3185870349407196, -0.9148372411727905, -1.0927757024765015, -0.616710901260376, -0.8535870909690857, -1.5967979431152344, -0.11069408804178238, -1.1351255178451538, -0.0020084665156900883, -0.04788645729422569, 0.8616244792938232, 0.6959834694862366, -0.39656707644462585, -0.05836717411875725, 1.1016850471496582, 0.06147607043385506, 0.2971426546573639, -1.304789662361145, 0.3420027792453766, 0.050900742411613464, -1.7323691844940186, 0.36015453934669495, 0.048919521272182465, 0.4854687452316284, 1.0621389150619507, 1.8157570362091064, 1.7972290515899658, -0.8064988255500793, 0.6780716776847839, -0.4729122221469879, -1.4713574647903442, -0.5051848888397217, -1.0372395515441895, 0.5696763396263123, 0.12745898962020874, 0.9660128951072693, 0.3983234167098999, 0.4226354956626892, 0.8916734457015991, 1.535215139389038, -0.9561341404914856, 0.673143744468689, -1.0126776695251465, -0.5117250084877014, -0.14732243120670319, 1.2638576030731201, -0.3048804700374603, 1.05447518825531, 1.5500032901763916, 0.25757789611816406, -1.769997477531433, 1.2188295125961304, 0.20550069212913513, -1.8550556898117065, 0.295355886220932, -0.4347079396247864, -0.9195588231086731, -0.8253036141395569, 1.0623431205749512, 0.9015761017799377, 0.8451656103134155, -1.1489408016204834, 1.3980216979980469, -0.8995349407196045, -0.7488016486167908, -1.2860372066497803, 0.33253195881843567, -1.5819644927978516, -2.2237870693206787, -0.9498149156570435, 0.5012747049331665, -0.3495197296142578, 1.1312867403030396, 0.8015155792236328, -1.031969428062439, 2.686735153198242, 1.2912142276763916, -1.2994076013565063, 1.098151683807373, -2.210399627685547, 0.10473079234361649, 0.9407598376274109, 0.8986210823059082, -1.1092205047607422, -0.46131953597068787], [-0.6998487710952759, 1.4053711891174316, -0.10365496575832367, 0.20281143486499786, -0.11566247791051865, 0.7809508442878723, -1.9959049224853516, 0.878686249256134, 0.1817408800125122, -0.14731143414974213, -1.9275951385498047, -1.4103050231933594, 0.7816310524940491, -1.8647255897521973, -0.8866485357284546, -1.9308717250823975, -0.8634242415428162, 1.9230647087097168, 0.6287906169891357, -0.22290943562984467, -0.9468474984169006, 0.8028512001037598, -2.125312089920044, -0.44393181800842285, -1.2299160957336426, 0.2852283716201782, 0.8603271842002869, -0.41304919123649597, -1.7676209211349487, 1.341443657875061, -0.35065218806266785, -1.1196602582931519, 0.44541576504707336, 0.3083934783935547, 0.010480055585503578, -0.15754982829093933, -0.7115486264228821, 0.6346902847290039, -1.5255482196807861, -1.335634469985962, -0.4050184190273285, -0.8103798031806946, 0.5691637992858887, 0.9385033845901489, -1.8873103857040405, 0.6512391567230225, -0.060618091374635696, -0.815907895565033, 0.03695777803659439, -0.6955147385597229, 0.9533721804618835, -1.2873166799545288, -0.5434709787368774, -0.05850113555788994, -0.41512933373451233, 0.4991835355758667, 2.191615104675293, -0.8362563848495483, 0.05114630609750748, 0.6767656207084656, -2.339104413986206, -0.3806120753288269, 1.0247434377670288, 1.3354408740997314, 0.17430509626865387, -1.502954125404358, -0.6968451738357544, 0.6113733053207397, 0.2772267460823059, 0.7931495904922485, -1.5158113241195679, 1.6603257656097412, 0.7333030700683594, -1.3328545093536377, 1.3231171369552612, -0.674625039100647, -0.6463735699653625, 0.4574722349643707, 0.5043583512306213, 0.07805026322603226, 0.21763275563716888, -1.0894168615341187, 0.6111785173416138, 0.7665835618972778, 0.6566365957260132, 1.1291799545288086, 0.2590227425098419, 0.5836275815963745, -1.902401328086853, -0.47084492444992065, 0.28386470675468445, -0.2604503035545349, 1.9340163469314575, 0.52462238073349, -0.7966696619987488, -1.469448208808899, -1.3356877565383911, -0.8648691773414612, 1.2515151500701904, 2.1756489276885986, 0.42894047498703003, -0.8558012247085571, 2.1409406661987305, 1.4941102266311646, 0.8147186636924744, 1.0115398168563843, -0.8549334406852722, 0.3066958785057068, -0.043760888278484344, -0.09598841518163681, 0.8938295841217041, 1.0649129152297974, 0.25011155009269714, -1.634750485420227, -0.8606868386268616, -0.5325905680656433, 0.45365163683891296, 1.8087632656097412, -0.8142224550247192, 1.3486781120300293, 0.34783506393432617, 0.4405117630958557, -0.8759236931800842, 1.0473228693008423, -1.5359314680099487, 0.1320706158876419, 0.2005273699760437, -1.0235849618911743, -0.7131527066230774, 1.0104591846466064, 0.47235211730003357, -0.5960087776184082, -0.5793303847312927, 1.4945052862167358, -1.5530520677566528, -0.7377084493637085, 0.08463431149721146, 0.7201252579689026, -0.9153789281845093, 0.021787747740745544, 0.1326255351305008, -0.12591588497161865, -0.15740995109081268, -1.5174663066864014, -0.44021180272102356, -0.4560755789279938, -0.5294654965400696, -1.4403201341629028, 0.49013155698776245, 1.528791069984436, 0.6234345436096191, 0.8444158434867859, 1.5247993469238281, 1.0761991739273071, -1.9433250427246094, -0.6745417714118958, -0.6501076221466064, -0.8911589980125427, -2.1774299144744873, 1.8492323160171509, 0.6713424921035767, 0.638027012348175, 0.8603216409683228, -0.8954241871833801, -0.0927923396229744, 1.6119093894958496, -1.93788743019104, -1.0715363025665283, -0.38607025146484375, 1.7823526859283447, 0.522800624370575, -1.5188312530517578, 0.5155287384986877, -0.10084984451532364, -0.004058884456753731, -0.21040749549865723, 0.9146255850791931, 0.3778138756752014, 0.2063615620136261, 0.8981217741966248, 0.42077046632766724, 0.37110018730163574, -1.962550163269043, 0.6393738389015198, 1.1668388843536377, 0.7513928413391113, 0.05804923176765442, 0.8570745587348938, -0.899395763874054, 0.2253139168024063, 0.741956889629364, -0.6034847497940063, 0.0785093903541565, -0.5311388969421387, 1.8393809795379639, -0.9184762835502625, -0.29658183455467224, -0.09577476233243942, -0.6254125237464905, 1.2583913803100586, -0.3580666482448578, -2.6653072834014893, 0.21221444010734558, 0.31690236926078796, -0.6464082598686218, -0.5979140400886536, -0.038871049880981445, 0.9533493518829346, 0.8448946475982666, 0.8952538371086121, -1.5558676719665527, 0.8669939637184143, -0.26962873339653015, 0.41757017374038696, -0.4115927815437317, -0.4443838596343994, -0.45127326250076294, -0.0043569402769207954, -0.12314916402101517, 0.04528016597032547, 1.1429502964019775, -0.5062614679336548, -1.393264889717102, 0.034450020641088486, -0.7163812518119812, 1.0520222187042236, -0.2550462484359741, -0.24989071488380432, 0.34501269459724426, -0.8255432844161987, 1.6286687850952148, -0.5338617563247681, 1.0142959356307983, -1.1720138788223267, -1.8488911390304565, 1.585741400718689, 0.5332301259040833, 0.5377734303474426, 0.2914429008960724, -0.7147448658943176, 2.177180528640747, 0.4481884837150574, -0.6055920720100403, 0.2255481779575348, 0.31218037009239197, -0.6537689566612244, 1.231638789176941, 0.8695974349975586, -0.32044267654418945, 0.4552288055419922, -0.43140003085136414, -2.027987480163574, 0.7584966421127319, 0.7249717116355896, -0.18972431123256683, 0.17783091962337494, -0.34282878041267395, 0.16340447962284088, 1.5070033073425293, -0.5326005816459656, 0.9146339893341064, -0.809080183506012, 0.7434762120246887, -0.5254957675933838, 1.9234914779663086, 0.15298998355865479, 0.4357897639274597, 1.5588042736053467, 0.04928739368915558, 0.2273884415626526, 0.7232959866523743, 0.20916806161403656, 0.1753072887659073, 0.8887355327606201, -1.5058454275131226, 0.6251994371414185, 0.9798842668533325, -0.7330539226531982, 0.0004458836338017136, 0.16364794969558716, 0.9844913482666016, 0.059668928384780884, 0.1341519057750702, 0.6943111419677734, 1.185276985168457, -0.040838807821273804, -1.9053326845169067, 0.14120686054229736, -0.1108938455581665, 0.20465046167373657, 0.9246461987495422, 0.22609910368919373, -0.19697746634483337, 0.3913297951221466, 0.26834896206855774, 0.32173675298690796, -1.083603858947754, -0.6203005313873291, -0.6627015471458435, -0.2659520208835602, 0.31100425124168396, 0.03640849143266678, 0.5436471700668335, 0.2922521233558655, -0.40975698828697205, -0.15143723785877228, 1.6290671825408936, -0.96148282289505, 0.34197208285331726, 0.7532407641410828, 1.1842005252838135, -0.048938822001218796, -0.19796954095363617, -1.8865368366241455, 0.22719265520572662, 0.658909022808075, -0.1804439276456833, 0.9297435283660889, 0.13807861506938934, 1.928398609161377, -0.3001769483089447, 0.23377688229084015, 0.33596906065940857, -0.38200321793556213, -0.9283397793769836, 0.2603095471858978, -1.4077168703079224, -1.9794223308563232, -0.06914414465427399, 0.2894604206085205, 0.07357336580753326, 0.4925931394100189, 0.9545491933822632, 0.743461012840271, -0.44480541348457336, 0.04100685566663742, -0.1568520963191986, -0.9404244422912598, -0.3308408260345459, -0.057462312281131744, 0.2445705682039261, 0.6028302907943726, 0.7351904511451721, 0.20743700861930847, -0.24106386303901672, 0.07953693717718124, 1.4919975996017456, 1.306985855102539, -0.555688738822937, -0.14077414572238922, -1.237382173538208, -0.32716256380081177, -1.074690580368042, 0.9350804686546326, -0.22229282557964325, -0.6163308620452881, -0.10671349614858627, -0.35097578167915344, 0.6805142164230347, -0.6877729892730713, -0.3531021773815155, -0.01180268730968237, 1.667365550994873, -0.43817204236984253, -1.4228402376174927, 2.188868999481201, 0.7247776985168457, -0.41713905334472656, -0.10428465157747269, -0.9932659268379211, 1.5976749658584595, 1.0280230045318604, 0.15463940799236298, 0.36080601811408997, 0.5955601930618286, -0.09250748157501221, 0.02622251585125923, 1.1454325914382935, -0.9773176908493042, -0.670181393623352, -0.13394518196582794, 0.007113678380846977, -0.6016265153884888, -1.5306326150894165, 0.7106929421424866, -0.13012652099132538, -0.2687203586101532, 1.1224490404129028, -0.6971246600151062, -1.42543363571167, 0.09569935500621796, -1.1739857196807861, 0.5413097739219666, 0.30926376581192017, 0.11881591379642487, 0.773263156414032, -0.8521873354911804, -1.1797199249267578, -1.6833324432373047, 0.7497121691703796, 0.5079132914543152, -0.6594582200050354, 0.3138844668865204, 0.18079298734664917, -0.020807228982448578, 1.7033023834228516, -1.1522918939590454, -1.0888453722000122, -1.2669308185577393, -0.6675385236740112, 0.33416593074798584, -0.3824634253978729, -0.39605870842933655, -1.2652602195739746, 0.2104927897453308, 0.10539654642343521, -0.7542462348937988, -0.8920990824699402, -1.759655475616455, -1.1418299674987793, -0.7630613446235657, 0.669581949710846, 1.2957251071929932, 0.1863592565059662, 0.5652710199356079, -1.0547419786453247, -0.1420917510986328, -0.42560288310050964, 0.6809402704238892, 0.7353317141532898, -1.1262856721878052, -0.23589199781417847, 0.1893889605998993, 0.42584702372550964, 0.08675475418567657, -1.1385257244110107, 0.025289759039878845, -0.08160696178674698, -0.12051913887262344, 2.0219340324401855, -2.1454617977142334, 1.509559988975525, 0.6613274216651917, 0.6651817560195923, 0.5715858340263367, 0.16987501084804535, -1.128228783607483, -0.9434864521026611, -0.17735406756401062, 0.1901676505804062, -0.27429041266441345, -0.43943002820014954, -1.7275421619415283, 0.9144173860549927, 0.8907514810562134, -1.685586929321289, -0.40710267424583435, -0.8664197325706482, -0.015997406095266342, 0.6553440690040588, -0.31241777539253235, -0.2984006702899933, 0.8190727233886719, -1.2431007623672485, 0.22998206317424774, 1.1382216215133667, 0.7104403376579285, -0.1948707401752472, 0.6487082242965698, -1.2849559783935547, -0.9752640724182129, -0.14530718326568604, -1.4663037061691284, -1.037691593170166, 0.054866887629032135, 0.5345001220703125, -0.04346177726984024, 0.8219719529151917, 0.1692458987236023, 0.19352932274341583, 0.38281968235969543, -0.9066935777664185, -0.45379236340522766, -0.5252779722213745, -0.7906298041343689, -0.030890148133039474, 0.11862920224666595, -0.26451292634010315, 0.6254079937934875, -0.05448460951447487, -0.6759194731712341, -2.507883071899414, 0.8919675946235657, -0.1621612161397934, -0.6871598362922668, -0.6413034796714783, -0.14577986299991608, 0.4565728008747101, -1.8433558940887451, 0.8682984709739685, 1.4562079906463623, 0.03268950432538986, -0.43272292613983154, -0.26668521761894226, -2.4513556957244873, -1.7639278173446655, -0.6754031777381897, -1.7670785188674927, -1.0015537738800049, 0.4914455711841583, 0.3370479345321655, 0.5604374408721924, 0.11603707820177078, 0.5694318413734436, 0.6720085144042969, 1.1593903303146362, -1.2846533060073853, -1.3639607429504395, -0.04984061047434807, -0.7634779214859009, -0.35925307869911194, -0.3812876343727112, 1.4099880456924438, 0.5562123656272888, -0.6107099056243896, -1.6669055223464966, -1.3942431211471558, -0.5470433235168457, 0.1803905963897705, 1.2333428859710693, 0.7670527696609497, -1.0170396566390991, 0.9813288450241089, -0.577553927898407, -0.02908346801996231, 0.6645033359527588, 1.469570517539978, 0.4577004611492157, -1.2925323247909546, 1.0656063556671143, -0.021126801148056984, 0.2327289581298828, -0.6753154993057251, -1.6776399612426758, -0.6420542597770691, -0.7855470776557922, 0.5617519617080688, 0.01047036424279213, -0.03429264575242996, -0.6374070048332214, 0.06646578758955002, 0.4503348767757416, -1.843015432357788, 0.34505340456962585, 1.204525351524353, -1.067042350769043, 1.242759346961975, 0.8053107857704163, 0.7607142925262451, 1.2140523195266724, -0.6225930452346802, -1.1343779563903809, 0.9483288526535034, -0.8608419299125671, -1.3145161867141724, 0.44617825746536255, -0.00738026387989521, -0.9508246183395386, -0.5023163557052612, 0.5170304775238037, 0.45228126645088196, -0.04636203125119209, -0.8453609943389893, 1.531813621520996, -0.3394385576248169, 0.8988160490989685, 0.5796849131584167, -0.7993050813674927, 0.5437249541282654, 1.5080021619796753, -0.6694575548171997, -0.12691736221313477, 0.5494822859764099, 1.5183404684066772, -0.2613460123538971, 0.462198942899704, 1.114217758178711, -0.768088161945343, -0.506197452545166, 0.6832283735275269, 0.10089965164661407, -0.13780610263347626, -0.7408806681632996, -1.0341517925262451, 1.9482381343841553, 0.09663937985897064, 0.19122251868247986, -1.3479241132736206, -1.0735284090042114, -0.4834083318710327, 0.07120855152606964, -1.8184126615524292, -0.3082776367664337, -0.633080780506134, -1.0821456909179688, -0.14344322681427002, 0.9907451868057251, -0.10928282886743546, -1.2376326322555542, -0.4533810019493103, 0.08079826086759567, -0.8382964730262756, 0.25374144315719604, -0.49173450469970703, 1.0082217454910278, -1.211212158203125, 0.762614369392395, -0.984989583492279, -0.6457483172416687, 0.12255723029375076, 1.4881938695907593, -0.03210261091589928, 0.2694571018218994, -0.5670090317726135, -1.1027389764785767, 0.13486988842487335, -0.3898462951183319, 0.1077674850821495, 0.8327569961547852, -1.690094232559204, 0.19093599915504456, 0.15916666388511658, -0.5849981307983398, 0.20526093244552612, 0.6331087946891785, -1.5743122100830078, -1.4528110027313232, 0.13401362299919128, 1.107268214225769, 1.1098904609680176, -1.1805146932601929, 1.0917737483978271, -0.24206015467643738, -0.71897292137146, -1.835774540901184, 0.9902385473251343, 1.7807731628417969, 0.2484704554080963, -0.4146963953971863, -0.20814363658428192, -0.33796828985214233, 0.501015841960907, -0.6039711236953735, 0.5258151888847351, 1.189065933227539, -1.2041728496551514, 0.21416893601417542, -0.09762027114629745, 1.1976277828216553, -0.4096727967262268, -0.501630961894989, 0.8487098217010498, 1.2251681089401245, -0.9049959182739258, -1.5266731977462769, -0.3547744154930115, -0.875386655330658, 0.8064472675323486, -0.44918227195739746, -1.0875613689422607, -0.09526285529136658, -1.5895085334777832, -1.6706347465515137, 1.3523664474487305, -0.670586109161377, -0.2659491002559662, -0.09956787526607513, -0.5113411545753479, -0.2952277958393097, 1.825913667678833, 1.1770405769348145, -0.22993670403957367, 0.03412282094359398, -0.1294051557779312, 0.20694264769554138, 0.4827592968940735, 1.1002146005630493, -0.7358020544052124, -0.7040829062461853, 0.1674668937921524, 0.1830909103155136, -0.2548002302646637, -0.5913029909133911, -0.0690944716334343, -1.3855819702148438, 0.35164785385131836, 0.5743898749351501, 0.08900684118270874, -0.643728494644165, 0.138549342751503, -0.8308903574943542, -0.01579240895807743, -0.25520044565200806, -0.9875766634941101, -0.9656941294670105, 0.9756362438201904, -1.2849009037017822, 0.2612840235233307, -0.09232360869646072, -0.851128876209259, -0.5801519155502319, 0.7995815277099609, -0.7087990045547485, 1.409233808517456, 0.4314851760864258, -1.4103116989135742, 1.8800339698791504, -0.04991352930665016, -0.058195535093545914, -0.018103428184986115, 0.9802600741386414, 0.2906830608844757, 1.7925652265548706, -0.4936921298503876, 0.32295578718185425, 1.1906074285507202, -1.6182852983474731, -0.22251103818416595, 2.1197304725646973, 0.28942903876304626, 1.6868844032287598, -0.9177232980728149, -0.19719025492668152, 0.45840030908584595, -2.4800117015838623, 0.2573070526123047, 0.17769184708595276, 1.9600385427474976, 0.49474450945854187, 1.2947975397109985, -0.20079122483730316, 1.7936575412750244, -1.225810170173645, 0.518044650554657, -0.49446889758110046, -0.36412152647972107, -0.4737185835838318, -0.055554766207933426, 0.09680833667516708, -0.10694567859172821, 1.0262125730514526, 1.0484018325805664, 1.3643189668655396, 1.2033933401107788, 1.9118901491165161, -0.3689388036727905, -0.07821313291788101, 0.9786072969436646, 0.9299925565719604, 0.8561439514160156, -1.0710060596466064, -0.19982433319091797, 0.2987963557243347, -0.8188310265541077, -0.2612544000148773, -0.5159332156181335, 1.8626375198364258, -0.7896760702133179, 1.056260108947754, -1.4335687160491943, -0.038116104900836945, -0.5999431014060974, 0.23074232041835785, -1.2427186965942383, 0.3764956295490265, -0.5022352337837219, 0.5035633444786072, -0.20203499495983124, 0.36139941215515137, -0.46270206570625305, 0.8715647459030151, 1.3166382312774658, -0.16158878803253174, -1.5750306844711304, 0.06789961457252502, 0.45199471712112427, -0.5055844783782959, 0.2059752196073532, -0.8848615884780884, -0.044705722481012344, -0.02711154893040657, 1.139076590538025, -0.395292192697525, -1.0285887718200684, 1.208032250404358, -1.9060108661651611, 1.510377287864685, 0.9257127046585083, -0.6465578079223633, 0.2239450216293335, -0.3611612915992737, -0.836267352104187, -0.8410800099372864, 1.75199556350708, -0.626207709312439, 1.251017451286316, -0.5608356595039368, 0.1320061832666397, -0.9818464517593384, 0.8464956283569336, 0.5383439660072327, -0.9514899849891663, -1.4269838333129883, -0.2401825338602066, -0.2995704412460327, -0.3447234034538269, -0.036066580563783646, 0.6489388942718506, 0.043688464909791946, -1.8827241659164429, -1.7863051891326904, 0.536309540271759, -0.03326603025197983, -0.5908485054969788, 0.9841464161872864, -2.0062291622161865, -0.5720731616020203, 2.081913948059082, 0.148651584982872, -0.5847155451774597, -0.6813392639160156, -0.1528450846672058, 0.971331000328064, -1.0577378273010254, -0.9773969650268555, 0.8907859921455383, -0.6436712741851807, 0.07049121707677841, -0.26784875988960266, 0.6440061330795288, 0.5955798625946045, -0.7781381011009216, 0.11735152453184128, 0.005417658947408199, -0.43910613656044006, 1.0414048433303833, -0.5748488306999207, 0.6738906502723694, 1.5923200845718384, -0.22441403567790985, -0.5191814303398132, 1.0642220973968506, -0.9147534966468811, -0.8172934651374817, 0.13144829869270325, -1.069535732269287, -1.0150210857391357, -1.815947413444519, -0.8698770999908447, 0.4433571696281433, -1.479737639427185, -0.7756637930870056, 0.43685874342918396, -0.3119840621948242, 0.11626271158456802, -0.18175716698169708, 0.32587721943855286, 0.8311758637428284, 2.7155532836914062, 0.41692739725112915, 1.1906402111053467, 0.5106247067451477, 1.3176870346069336, -2.266479253768921, 0.1736014485359192, 0.44546082615852356, -1.4000040292739868, 0.4997442066669464, -1.5595918893814087, 0.8272904753684998, -0.02660515531897545, 0.7367814779281616, -0.7123244404792786, -1.3988264799118042, -1.1615333557128906, 1.1926162242889404, 0.0111541161313653, -0.34576141834259033, -0.41168704628944397, -0.13566423952579498, -0.8517288565635681, -0.21912436187267303, 0.22478137910366058, -0.35570865869522095, -0.9235109090805054, -1.226971983909607, 1.1275895833969116, -0.33440926671028137, 0.7656315565109253, 1.6145187616348267, 0.9780842065811157, 1.3172825574874878, 0.9349581003189087, -0.7022935748100281, -0.5294241309165955, 0.2756463587284088, -0.7140378952026367, -2.867795944213867, 0.4811041057109833, -0.02055034227669239, 0.9502869248390198, -0.17274922132492065, -0.6294195055961609, -0.25217998027801514, 0.7036223411560059, -0.46101659536361694, -1.2697783708572388, -0.8585105538368225, 2.925858736038208, -1.6443569660186768, -1.2365214824676514, 1.3735698461532593, -0.08608753234148026, 0.6034565567970276, -1.1921734809875488, -0.1456720381975174, -0.6618180871009827, 1.2082055807113647, 1.0474135875701904, -0.844652533531189, 0.30356189608573914, 1.1200743913650513, -0.7736048698425293, -1.6527951955795288, -1.7351959943771362, 0.8449133038520813, -2.7021868228912354, -0.3187505900859833, -0.09578859061002731, 0.9065945148468018, -1.6001683473587036, -0.05222658812999725, 0.6569237112998962, 1.0098416805267334, 0.038003720343112946, -0.3930748999118805, 1.1244374513626099, 0.3299460709095001, -0.378023624420166, 0.28794392943382263, 1.317247748374939, 0.8959469795227051, -0.963837206363678, -0.6100690960884094, 1.1187505722045898, 0.6184594631195068, -1.1371694803237915, 0.08635523170232773, -1.1156060695648193, 1.539088487625122, 1.7291947603225708, -1.3331142663955688, -1.1470489501953125, -0.6269542574882507, -0.2397378385066986, 1.5889793634414673, 0.17448922991752625, -1.4467917680740356, -0.9132155776023865, 0.5672733783721924, -0.2906446158885956, 1.4318760633468628, -0.9692450165748596, -0.21831659972667694, -0.10047503560781479, -0.8730478286743164, -0.14871840178966522, 1.6227283477783203, -0.13615384697914124, -0.9070146083831787, -0.3762056231498718, 0.021068064495921135, -0.9417291879653931, 0.07403668016195297, -0.4743594825267792, -0.7018671631813049, -0.4261997640132904, 0.9810085296630859, 0.37953701615333557, -0.6846590638160706, -0.7558408379554749, 0.08751776814460754, 0.16461226344108582, -0.42735376954078674, -0.4399626553058624, -0.13500787317752838, -0.4777817726135254, -0.32505273818969727, 0.28414756059646606, -0.7374518513679504, -0.024758130311965942, 0.6486688852310181, 0.33065900206565857, -2.0542826652526855, -0.7088842988014221, -0.1422707438468933, -0.5986450910568237, -0.8012647032737732, 1.1824792623519897, 0.48899248242378235, 0.3747326135635376, 0.7775341272354126, 1.5316412448883057, -1.0666300058364868, 1.8893868923187256, -1.606464147567749, 2.3774826526641846, 0.6187083125114441, 0.5496063232421875, 0.713554859161377, -1.810212254524231, -1.502097725868225, 0.7566033601760864, 0.09313531965017319], [-1.0760948657989502, -0.35536089539527893, -0.6050125956535339, 1.6937633752822876, 0.7888005971908569, -0.10559839010238647, 1.3517624139785767, 0.17503111064434052, 0.11557256430387497, -0.37422385811805725, -1.2395540475845337, 1.7778346538543701, -1.7613155841827393, -0.13241463899612427, -1.1532984972000122, -1.7537014484405518, -0.03403187543153763, -0.5627367496490479, 0.7974642515182495, 1.7441346645355225, 0.32004231214523315, 0.25364503264427185, -0.7439205050468445, -0.9954516887664795, 1.7434462308883667, 1.1654924154281616, 0.2910429537296295, -0.920564591884613, -0.2875334322452545, -1.0749497413635254, 0.2691906988620758, -1.0012248754501343, -1.5607901811599731, -1.9308907985687256, -0.6816219091415405, 2.1283962726593018, 1.3770604133605957, 1.2662701606750488, -0.11086054146289825, 0.48857581615448, 0.4751187264919281, 1.005411148071289, -0.045190248638391495, 0.8150613307952881, 0.12489921599626541, -2.384258508682251, -1.0760372877120972, 0.24337443709373474, 1.175254225730896, 1.2968788146972656, 1.128825068473816, 1.1187275648117065, -0.750073254108429, 0.8022737503051758, -0.29666849970817566, -0.5442885160446167, -0.4882696866989136, 0.3537427484989166, 0.295927494764328, -0.6319307088851929, -1.312089204788208, -0.9799560308456421, -0.35215428471565247, 0.7627217769622803, -0.2810802757740021, 1.1286495923995972, 1.1704667806625366, 0.8662655353546143, -0.34919196367263794, 0.7991784811019897, 0.9514210820198059, -0.7294489741325378, -0.4109315276145935, 0.16207166016101837, -0.08009102940559387, -0.393889456987381, 0.3072075843811035, 0.6224115490913391, 0.4072333574295044, 0.4649513065814972, -0.889094889163971, -1.8235212564468384, 0.40725553035736084, 0.04192902892827988, -0.5291233658790588, -0.09244697540998459, 1.1497384309768677, 0.0434221550822258, 0.1350328028202057, 1.3669838905334473, -0.8479082584381104, 0.29787349700927734, 1.4756159782409668, -1.404738187789917, -1.831587553024292, 0.7952238917350769, 0.1906578242778778, 1.049014687538147, 1.6243237257003784, 1.055649995803833, -0.15904225409030914, 0.4593505263328552, 1.0529556274414062, -1.5302462577819824, -0.2481108456850052, -0.6940037608146667, 0.9724853038787842, 0.7687514424324036, -1.1142102479934692, 1.4763305187225342, 0.7394229173660278, -0.6270797848701477, 0.6859044432640076, 0.2913229465484619, -1.2859622240066528, 0.16405420005321503, -0.3397255539894104, 0.7535936236381531, 0.6914860010147095, -0.8759166598320007, -0.12755116820335388, -0.7810640335083008, -0.38705822825431824, -1.3283339738845825, 1.5880247354507446, -2.0437843799591064, -0.5833371877670288, -0.4963115155696869, 2.8161728382110596, 1.417109489440918, -0.2948263883590698, -0.4253782033920288, -1.4061216115951538, 0.02386276051402092, -0.9658634066581726, 0.8677618503570557, -0.58226078748703, 0.05336777865886688, 0.28196147084236145, -0.218546524643898, 0.21702472865581512, 0.16535010933876038, -0.9004809856414795, 1.2768770456314087, 0.6576072573661804, -1.2910586595535278, -0.6499178409576416, -0.3470548987388611, -1.9368896484375, -0.7420978546142578, -1.0792726278305054, -0.48052656650543213, -0.04714063182473183, 0.293649286031723, 1.4255144596099854, 0.8432924747467041, 0.6584998965263367, -0.26051509380340576, -0.9573362469673157, 0.42564842104911804, 0.19735337793827057, -0.2546592354774475, -0.0057700881734490395, 0.14404816925525665, -0.42296403646469116, -0.6350440979003906, -0.4806114137172699, 0.6964544653892517, -0.22163279354572296, -0.05393946170806885, -1.6517219543457031, -0.16902290284633636, -0.4298914074897766, -0.9270632266998291, -0.0591307133436203, -0.8803560137748718, 1.4951939582824707, 1.3316254615783691, 0.07044950127601624, -0.05561909079551697, -0.6810663938522339, -1.6897797584533691, -0.11461298167705536, 0.5884379148483276, 0.3095942437648773, -0.9437860250473022, 0.10029394179582596, 0.22926197946071625, 0.6980869174003601, 0.6548587083816528, -1.631517767906189, 1.380486249923706, 0.2787851393222809, 0.8294093608856201, 0.570280134677887, -0.6815177798271179, 1.1908098459243774, -0.6800230145454407, -1.8592888116836548, 1.3311275243759155, 0.6783151626586914, -1.607042908668518, 0.20128396153450012, 0.3582340180873871, -0.9789307713508606, -1.9332913160324097, 0.06929899007081985, 0.4225713610649109, -1.4587433338165283, 0.18196827173233032, 0.5658318996429443, 1.3139899969100952, 0.3126252591609955, -0.18718957901000977, -0.4602976143360138, -1.8321179151535034, -0.3743780553340912, 1.5634487867355347, -0.252109169960022, 0.3964308798313141, 0.5423157215118408, 0.2146536409854889, 0.7532171010971069, 0.02233232744038105, 0.8326458930969238, -0.6106442809104919, -0.9489613175392151, 1.6488357782363892, -0.44387540221214294, 0.1914702206850052, 1.026240587234497, 0.16240589320659637, -1.0931414365768433, -1.5918821096420288, 1.535416841506958, 0.3725968897342682, -1.6122756004333496, 0.23300525546073914, -1.5178197622299194, 0.1312273144721985, 0.515805184841156, 0.3990987241268158, 0.5184463858604431, 0.22005200386047363, 1.3005198240280151, -0.3036558926105499, 1.3245810270309448, -1.854055643081665, 1.7764599323272705, -0.39047813415527344, 0.8475376963615417, 2.0102174282073975, 0.6104224920272827, 0.5541701912879944, -0.3270803391933441, -1.4903002977371216, -0.4473867416381836, -1.0139968395233154, -1.306381106376648, 0.6713791489601135, -0.7713980674743652, -1.5796371698379517, -0.17108303308486938, 0.27986353635787964, -1.495008945465088, -0.4734397530555725, -0.1638399213552475, -1.0750740766525269, 2.3060498237609863, 0.7330434322357178, 0.275601863861084, 0.5563072562217712, -0.873688817024231, -0.9472529888153076, 0.07541593164205551, -1.3354263305664062, 0.7571907043457031, 0.5696973204612732, -0.5748550295829773, -1.8020416498184204, 0.45191678404808044, 0.3695020377635956, -2.0657925605773926, -0.962509036064148, -0.6861024498939514, 1.8054234981536865, 0.5706668496131897, -0.12952740490436554, -0.4693107008934021, 0.7095891833305359, 1.7146559953689575, -0.9867565035820007, 0.39668309688568115, -1.6830178499221802, -0.3909491002559662, -0.40699249505996704, -0.9582778811454773, -0.9725005030632019, 0.29080909490585327, 1.6951466798782349, -0.2859874367713928, -2.121537208557129, -0.3428504765033722, 0.3780887722969055, 1.3637181520462036, 0.6747704744338989, 1.1429271697998047, 0.17680831253528595, -0.25816312432289124, 1.4265462160110474, 1.4101232290267944, -0.1319369226694107, 1.1473606824874878, 0.2844351828098297, -1.2768921852111816, -1.6043204069137573, 0.3956059217453003, -0.4602476954460144, -0.6388386487960815, 0.8992725014686584, 0.8500971794128418, 1.2249972820281982, 0.046118803322315216, -0.3619324266910553, -0.16800014674663544, -1.5494295358657837, -1.110766887664795, -0.4148010313510895, -0.629672110080719, -0.11940953880548477, 1.4110549688339233, 0.31066176295280457, -0.4381246864795685, 0.9566192030906677, -0.3831409215927124, -0.13232605159282684, 2.146275043487549, -0.648098349571228, 0.2266964316368103, -1.6365681886672974, -0.4331156611442566, 0.4056135416030884, 1.252988576889038, -1.3234107494354248, 1.0741437673568726, -0.8982436060905457, 0.6318389177322388, 0.8033400177955627, 0.7061649560928345, 0.21670736372470856, 0.2931157350540161, -0.2885423004627228, -0.9273529052734375, 0.016456732526421547, 0.06469205766916275, -1.3157989978790283, 0.7066873908042908, -0.4017728865146637, 1.6040936708450317, -0.3569640815258026, 1.2068345546722412, -0.39083486795425415, -0.332078754901886, -0.5671259164810181, -1.3193342685699463, 1.2161219120025635, 0.8928292393684387, -1.2994177341461182, -1.1193479299545288, 0.207496777176857, -2.941642999649048, -1.5971394777297974, -0.4370085895061493, 0.18845373392105103, 0.2580149471759796, -1.4456268548965454, -1.7844716310501099, 1.2393274307250977, -1.320095419883728, 1.139933705329895, -0.5222521424293518, -1.1914299726486206, -0.3084762692451477, 0.2216585874557495, -0.6435007452964783, 0.38057321310043335, 0.03384266793727875, -0.906327486038208, 0.45647841691970825, 0.26278388500213623, -0.5222795009613037, -1.0198285579681396, 0.5626416206359863, -0.9474325180053711, 0.7759690284729004, 2.2721176147460938, 0.07130104303359985, 0.16696244478225708, -0.29289621114730835, -0.3548465669155121, -0.007022975943982601, 0.9662110209465027, 0.21475690603256226, -0.6338323950767517, -1.6305783987045288, -0.24168120324611664, 0.05550514534115791, 0.725480318069458, 0.5452368259429932, 1.6204097270965576, 0.7946365475654602, -1.462003231048584, -0.5799054503440857, -1.9187966585159302, -0.7933026552200317, -1.135644793510437, 0.2502801716327667, -1.0359370708465576, 0.9021599888801575, 0.37068331241607666, -0.8048966526985168, 0.3091544508934021, -1.266175627708435, 1.7040276527404785, -0.461629718542099, -0.3401321768760681, -0.6360422968864441, -0.02331831492483616, -0.8707671165466309, 0.3066719174385071, 0.32109734416007996, 0.3817867338657379, -0.21992284059524536, 0.505069375038147, 0.6666284799575806, -1.775827169418335, 1.1843113899230957, 1.0748456716537476, 1.3092191219329834, 1.5114067792892456, 0.2537388801574707, 0.9254814982414246, 0.10620807856321335, 0.254901260137558, -1.1236132383346558, -0.7340700030326843, -0.46412596106529236, 1.068259835243225, -0.09863172471523285, -0.4750373959541321, 1.5036002397537231, -1.182598352432251, 0.6265225410461426, 1.648230791091919, -0.24348486959934235, -0.1263585090637207, -0.26697051525115967, -1.6564635038375854, -0.6823033690452576, -0.8435903787612915, 1.1102993488311768, 0.5785733461380005, 0.6048856973648071, -0.2972579002380371, -0.5001422166824341, -1.0500587224960327, -0.047231223434209824, -0.40398821234703064, 0.3730441927909851, 1.3951314687728882, -0.10778598487377167, 0.2755063474178314, 1.0227171182632446, -0.3518974483013153, -1.0913124084472656, -1.2755783796310425, -1.6374698877334595, -0.8186987042427063, 0.6758596301078796, -0.8183359503746033, -0.11795861274003983, 0.015325977467000484, -0.289515882730484, 1.5917208194732666, -0.6091940999031067, -0.6916764378547668, -0.36599430441856384, 0.8249415755271912, 1.715723991394043, 1.168184518814087, 2.2215328216552734, 1.6649011373519897, -0.5677884221076965, -1.2362140417099, -1.0350370407104492, 3.0831894874572754, -1.006473183631897, 0.20475083589553833, 0.8154096007347107, 1.312443733215332, 0.00926719419658184, 0.32286491990089417, -1.3184258937835693, 1.456120491027832, 0.2047622948884964, -0.23838764429092407, -0.2507810592651367, -0.07165298610925674, 0.321578711271286, -0.7648811936378479, 0.026043858379125595, 0.5828348994255066, 0.13489535450935364, -0.2622263729572296, 0.06801855564117432, -0.5354448556900024, -0.035522475838661194, 1.7040103673934937, -1.4599206447601318, -0.2894202768802643, -0.6826930642127991, 0.9844173192977905, 0.14851965010166168, 1.114662528038025, -0.6913930773735046, 0.13571833074092865, -1.1599531173706055, -1.908868670463562, 0.6818215250968933, 0.20484335720539093, -1.1032732725143433, -1.1365965604782104, -0.870795726776123, -0.04029737785458565, -1.4443950653076172, -1.9242382049560547, -0.559069037437439, 0.054545529186725616, -0.340637743473053, -1.7534019947052002, -0.3209120035171509, -0.8016834855079651, -0.14107882976531982, -0.7875359654426575, 0.739448606967926, 0.04146586358547211, 1.571054220199585, 0.48572003841400146, -0.9302693605422974, -1.2000420093536377, -0.9991969466209412, 1.443973422050476, 0.932861864566803, -0.0034800756257027388, 1.3014333248138428, -1.237267255783081, 0.3880695700645447, -0.729102611541748, -1.2750269174575806, 0.8947929739952087, -1.8451614379882812, 1.7171494960784912, 0.5069427490234375, 0.514445960521698, -0.9990077018737793, 0.4313589036464691, 0.3522065579891205, -2.2297306060791016, -0.6883590817451477, 0.03682998940348625, -0.5243626832962036, -0.43608996272087097, 0.280449241399765, 0.5502904057502747, 1.033126950263977, -1.5895957946777344, -0.37799325585365295, -0.636625349521637, 2.0652973651885986, -0.17181171476840973, 0.3788672387599945, -0.907663881778717, 0.879371702671051, 0.2889946401119232, -1.3403842449188232, -1.015622615814209, 0.5850863456726074, -1.1630831956863403, -0.3010006546974182, -1.6365793943405151, 0.2350524663925171, 1.2503503561019897, -0.16920286417007446, 0.06758426874876022, -0.18495935201644897, -0.36890023946762085, 0.7244915962219238, 1.4307836294174194, 1.366640567779541, 0.4295457601547241, 0.9158065319061279, 0.21186386048793793, -1.5601956844329834, 2.530460834503174, -1.3474467992782593, 0.9526681303977966, -0.7511447072029114, 0.0699903592467308, 1.4436439275741577, 1.360421061515808, 0.049394674599170685, 0.3103621304035187, -0.5373163223266602, -1.4886207580566406, -1.4497562646865845, -0.22635453939437866, -0.4532855451107025, 0.855094850063324, 1.3866254091262817, 0.42391538619995117, -0.5674228668212891, -0.4406067132949829, -0.3367178738117218, -0.6709896326065063, -0.8476874232292175, 1.20035719871521, -0.6366738080978394, -2.002345085144043, 1.4782339334487915, 1.0129977464675903, 0.37309011816978455, -0.13715098798274994, -0.44022345542907715, -0.25884169340133667, -1.217206597328186, 1.3571821451187134, 0.8048307299613953, -0.878746509552002, 1.5642204284667969, 1.4579949378967285, -1.095193862915039, 0.03733399510383606, 0.25607508420944214, 0.2228553295135498, -1.1843851804733276, 0.5306247472763062, 1.1979748010635376, -1.7937332391738892, 0.6651743650436401, -0.6900938153266907, -0.02927115000784397, -0.39506909251213074, 1.090180516242981, -0.903177797794342, -0.42328402400016785, 1.2426599264144897, 0.04009120166301727, -1.060267686843872, 0.33240363001823425, -1.0057308673858643, -1.9621599912643433, -0.6202166080474854, -0.48710981011390686, 1.401643991470337, 0.3435310423374176, -0.299803763628006, -1.3279180526733398, 0.7503089904785156, -0.02095849998295307, -2.526552677154541, -0.4406127333641052, -0.3238963782787323, 0.5994732975959778, -0.8330608010292053, -0.9040820002555847, 0.4995270073413849, 0.21563266217708588, 1.092523217201233, 0.37348708510398865, 0.2591499984264374, -0.06309668719768524, -0.8431276082992554, 1.979569911956787, -0.5192081332206726, -0.7592259645462036, 0.36059585213661194, 1.0429911613464355, -0.4335715174674988, 0.11909547448158264, -0.3828195631504059, -0.9643799066543579, 0.8991851210594177, -1.1421715021133423, -1.623498558998108, 1.9675121307373047, -0.4184250831604004, -1.3466273546218872, -0.5742645263671875, 0.5105189681053162, 0.9706607460975647, -0.06215642765164375, 0.40620511770248413, 1.1165622472763062, 0.15932928025722504, 1.4970412254333496, 1.2289719581604004, 1.2642453908920288, -1.1179975271224976, -0.6071568131446838, -1.0590453147888184, -0.3198687434196472, -0.9403533935546875, 0.7633125185966492, 0.0674237385392189, -0.28207632899284363, -0.6642938852310181, -0.3781183660030365, 2.66975474357605, 0.30742278695106506, -0.7372404932975769, -0.020785540342330933, 1.4352613687515259, -1.1031558513641357, 0.5319985151290894, -0.7060588598251343, 0.4774772822856903, 0.6776375770568848, 1.3687244653701782, 0.947123646736145, 0.46306806802749634, -0.29951900243759155, 1.1779714822769165, -1.064918041229248, 0.4067540168762207, 0.9604565501213074, 1.7410070896148682, -0.7903949618339539, -1.3322126865386963, 0.4756297767162323, 1.9603768587112427, -1.099670648574829, 1.9385251998901367, -0.01695108786225319, -0.544431209564209, -0.9388284683227539, 0.6437236070632935, -0.9155164957046509, -0.6095016598701477, -1.537039041519165, -0.1094031035900116, -0.6934072375297546, -0.4419776201248169, -0.6031595468521118, -0.08029846101999283, -1.0678328275680542, 1.7720224857330322, 0.9286434650421143, -0.8814992308616638, 0.04680362716317177, 0.5785698890686035, -2.2321736812591553, 0.7864148020744324, 0.0762048214673996, 0.3034234344959259, -1.3971892595291138, -0.2699790298938751, -0.07302634418010712, 0.2087169587612152, -1.1211503744125366, 1.8679319620132446, -0.3634185492992401, 1.2461391687393188, 0.01966775394976139, -0.3843945860862732, -0.9451400637626648, 0.8122351765632629, 0.10487423092126846, 0.143978551030159, 0.001265743630938232, -0.8030858635902405, -1.3486394882202148, -0.09860382974147797, -1.2310150861740112, 1.5404412746429443, 0.9849605560302734, 0.07731907814741135, 0.6244847774505615, 0.8962618112564087, -0.09435398876667023, 0.16070348024368286, -0.17924652993679047, -0.5940184593200684, -0.5061453580856323, -0.7021862268447876, -0.006282902788370848, 0.7703160643577576, 0.44691312313079834, 1.085368275642395, -0.34857290983200073, 0.7125225067138672, -0.5093995332717896, -1.0275192260742188, -0.5162961483001709, 0.011033676564693451, 0.2191789299249649, -1.5859713554382324, -0.6373744010925293, 0.571639895439148, -1.0926992893218994, -0.9822458028793335, 1.7319518327713013, 0.11095096915960312, -1.1087182760238647, 1.9118226766586304, -0.8869671821594238, 1.552366852760315, -0.5218451619148254, 0.5875614285469055, 0.15431545674800873, -1.3543286323547363, 1.413543701171875, -1.476631999015808, -0.08480840921401978, 0.34627360105514526, -0.17698734998703003, 1.5618512630462646, -0.6099798679351807, 1.6252461671829224, -0.1939925253391266, -0.5229853987693787, 0.11005853116512299, -2.4398553371429443, 1.494704008102417, -1.0598386526107788, -0.2791634500026703, 0.9799063205718994, -0.23876415193080902, -1.5163357257843018, -1.0737717151641846, -1.2672408819198608, -2.0483596324920654, -0.26690199971199036, -0.7349557280540466, -0.41022610664367676, -1.2075806856155396, 0.044954925775527954, -0.09211690723896027, -0.24370142817497253, -1.1309874057769775, -0.03520466387271881, 0.7821343541145325, -0.6203077435493469, -0.00846883188933134, 0.7114623785018921, 0.26188457012176514, -0.4629993736743927, -0.8033707141876221, -0.21774668991565704, 0.3543182909488678, 0.4162197411060333, 0.2763729691505432, -0.6311317086219788, 0.08908922970294952, -0.37525343894958496, 0.007095244713127613, -1.6831660270690918, -0.1133737564086914, -0.24126887321472168, -0.4403453469276428, -0.3572739064693451, 1.2948049306869507, -0.1456175446510315, 0.4336208403110504, 0.5893886685371399, -1.1502782106399536, -0.22321267426013947, 1.0360850095748901, 0.8529568314552307, -1.6067575216293335, -0.22412526607513428, 1.1660860776901245, -0.48755398392677307, 0.9912964105606079, 1.279619812965393, -0.7163791656494141, 0.8930666446685791, -1.7125507593154907, -1.288728952407837, 1.640982747077942, 0.7740721702575684, -1.1430509090423584, -0.40238285064697266, -1.570701003074646, 1.641350507736206, 0.6627265214920044, -0.10645262897014618, 0.5992160439491272, -0.9655208587646484, 1.5062388181686401, -0.7976215481758118, 0.9256260991096497, 0.5122212767601013, -0.42407935857772827, -0.8318573832511902, -0.204987570643425, 0.052831824868917465, -1.198379635810852, -1.0543161630630493, -0.5015048384666443, -0.5104656219482422, 0.3344970941543579, -0.06414447724819183, 1.339790940284729, -0.611524760723114, 2.8346052169799805, 1.161507487297058, 1.130670428276062, -0.4048967659473419, 1.4957858324050903, -0.040235720574855804, 0.17685000598430634, -0.543813943862915, -0.418020099401474, 0.05367524176836014, -0.30458149313926697, 0.29162976145744324, -1.1890671253204346, 1.7565184831619263, 1.1821852922439575, 1.0792502164840698, -0.07172707468271255, -0.1508638709783554, 0.10643067955970764, 1.0073060989379883, 1.5648270845413208, -0.6174629330635071, 0.06233667954802513, 1.125127911567688, -0.22657592594623566, -0.5376735329627991, 0.2837148904800415, -0.402736097574234, -1.3986021280288696, 2.705408811569214, 1.4100096225738525, -1.0310510396957397, -0.2911980450153351, -0.20534680783748627, 0.5337289571762085, 0.790008008480072, -1.5304579734802246, 2.10309100151062, -1.4185608625411987, 0.24670305848121643, 0.06756813079118729, 2.4590096473693848, -0.15894311666488647, -0.2901322543621063, 0.3937365710735321, -0.5765891075134277, -0.0864114761352539, -0.6728670597076416, 0.3837866485118866, 0.014465835876762867, -2.1136467456817627, 0.512861967086792, -0.24354441463947296, 0.36975130438804626, 0.27730968594551086, 0.1327972561120987, -1.649512529373169, 0.055503081530332565, -0.9264860153198242, 0.08217158913612366, 1.121701955795288, 3.335451364517212, 1.1641374826431274, -1.3505046367645264, -1.9845168590545654, 0.21435096859931946, 0.5671600103378296, -0.7378785014152527, 0.8782553672790527, -0.15744495391845703, -0.08049505203962326, -2.053391218185425, 0.25659140944480896, 1.1050463914871216, -1.295433759689331, 0.048483237624168396, -0.7320539355278015, -0.796392023563385, 0.06729314476251602, -0.7741001844406128, -1.532986044883728, 0.4285266399383545, -0.1129659116268158, -0.9007390141487122, -0.5649495124816895, 0.6513127088546753, 2.2837421894073486, -1.4073482751846313, 0.7702580094337463, -0.15925569832324982, 0.23748525977134705, -0.7799363136291504, -0.06928970664739609, 1.695994257926941, -1.0156258344650269, 1.0725328922271729, 0.09765799343585968, -0.12877270579338074, 0.19866812229156494, -0.5174540877342224, -0.7685117721557617, -1.233576774597168, 0.7077450752258301, 2.8075673580169678, -0.7070277333259583, -0.07672099769115448, 1.455330491065979, 1.2389053106307983, 0.15313886106014252, -1.6166573762893677], [-0.2474149614572525, -0.033062297850847244, -1.2915211915969849, -0.23698708415031433, 0.3772893249988556, 0.18757230043411255, -0.4887208044528961, 0.5026088953018188, 0.7834783792495728, 0.9998810887336731, -0.4310215413570404, 0.15697994828224182, -0.1448683738708496, 1.2206279039382935, -0.9485570192337036, -2.1016757488250732, -0.8692351579666138, -0.2195810228586197, 0.2038150131702423, 1.1230542659759521, 1.5203834772109985, -0.9341157674789429, 1.2481989860534668, -0.6647019982337952, -1.6410458087921143, -0.05340556427836418, 0.1045745387673378, 0.5976073741912842, 1.7929084300994873, -1.4240132570266724, -0.05776191130280495, -1.9130518436431885, 1.2554975748062134, 0.019130000844597816, 0.32619959115982056, 0.41091063618659973, -0.2954847812652588, -0.19724194705486298, 0.26068148016929626, 0.7520478367805481, 0.3188050389289856, -0.1133129671216011, -1.0570969581604004, 0.29993125796318054, 0.26819470524787903, -0.8002375960350037, 0.4221368134021759, 0.1939915418624878, 0.9831470847129822, -0.047443993389606476, 2.6538009643554688, -0.07710240036249161, -0.9809650182723999, 0.6016921401023865, -1.5112794637680054, 0.8867674469947815, -0.13503111898899078, 0.6802100539207458, -0.16926218569278717, -0.7624255418777466, 1.0050735473632812, 0.786076009273529, 0.34872961044311523, 1.294482707977295, 1.544357180595398, 0.5202702879905701, -0.10881680250167847, -0.3770129680633545, -1.5001500844955444, -1.4672459363937378, -1.0914450883865356, -2.020035982131958, 0.2995733916759491, -1.0554754734039307, 2.3903002738952637, 0.26195651292800903, 0.21374456584453583, -1.3645610809326172, 1.7375777959823608, 0.416079580783844, 0.5066797733306885, 0.20489487051963806, 0.696376621723175, -0.6326954960823059, 0.8510532379150391, -1.042726755142212, -0.29313600063323975, -0.4659789204597473, 0.15341299772262573, -0.685875654220581, -1.7606667280197144, -0.13020150363445282, 1.546560287475586, 0.027799459174275398, -0.017651250585913658, -1.0049591064453125, -1.0137999057769775, 0.7847623229026794, -1.0618207454681396, 2.057271718978882, 0.9363928437232971, 0.6262236833572388, -0.5830169916152954, -0.4001592993736267, -0.3481070101261139, -0.0060023292899131775, -1.022529125213623, -0.23780199885368347, 0.8210465312004089, 0.7031239867210388, 0.33321520686149597, 1.1829111576080322, 2.0367860794067383, 1.52558434009552, 1.4210795164108276, 0.43903473019599915, -0.570319652557373, -0.16626837849617004, 0.8721997737884521, 0.2472507357597351, 1.2930275201797485, -1.2162638902664185, -0.17307959496974945, 0.621415376663208, -0.42504140734672546, 0.36776241660118103, 0.17422649264335632, -0.2275121510028839, 0.8523781299591064, -0.5024884343147278, 0.2901484966278076, -0.8142539858818054, -1.2389189004898071, 0.12702499330043793, 0.391965389251709, 1.3522206544876099, 1.0465649366378784, -0.9210079312324524, -0.6883599162101746, -0.9136481285095215, -0.7407394051551819, 0.9246580600738525, -0.06881771236658096, -0.4545834958553314, -1.1100341081619263, 1.8411394357681274, -0.22687102854251862, 1.3532785177230835, 0.3276800513267517, -0.04589631035923958, -1.435912847518921, -0.7129356265068054, 2.3697545528411865, -1.462180733680725, -0.8525424599647522, -1.8292384147644043, -1.5047731399536133, -0.029192840680480003, 0.8720099329948425, 0.8343638777732849, -0.40235278010368347, -0.007672042585909367, 0.07241404056549072, -0.4232138991355896, -0.8134585618972778, 0.6472043991088867, 1.868332862854004, 0.1264185905456543, -0.4608878791332245, 0.15756045281887054, -0.20651279389858246, 1.5960954427719116, -0.7105262279510498, -1.255467176437378, 0.2287953794002533, -0.14266636967658997, -0.670689046382904, 0.42350178956985474, 0.6843100190162659, -0.15884649753570557, -0.34772437810897827, 1.511038064956665, -0.08769652247428894, -0.9467759728431702, -0.709933340549469, 0.9310848712921143, 1.9227715730667114, -1.0158698558807373, 1.0761222839355469, -0.695626974105835, -0.2803824245929718, 0.11365421116352081, 0.03217635303735733, 0.05852378159761429, -0.2914939820766449, 0.07889792323112488, -0.8672997355461121, -1.1188479661941528, 0.7687187194824219, -1.6056642532348633, 1.885636568069458, 1.7676180601119995, -0.2511778473854065, -0.08287147432565689, 0.10403739660978317, 1.574965476989746, -0.0009678188362158835, 0.8396531343460083, -0.2022693157196045, -2.477796792984009, 0.24600128829479218, 1.7429163455963135, 1.0824618339538574, -0.11273232847452164, -2.7777726650238037, -2.3782413005828857, -1.5015432834625244, -0.6798920035362244, 1.1203460693359375, 1.0557005405426025, 0.5190374255180359, 0.6767621636390686, 0.0867103859782219, -1.4850742816925049, -1.0518535375595093, 0.8473620414733887, 1.0057847499847412, -1.8141170740127563, -1.97276771068573, -0.8491032719612122, -0.0021721820812672377, -1.1847602128982544, 0.8446548581123352, -0.9658660292625427, -1.5108965635299683, 0.6554349660873413, 0.1650780588388443, -1.0335103273391724, 0.4033699929714203, 1.1942120790481567, 1.0932453870773315, -1.1213536262512207, 0.07184173911809921, -1.2371453046798706, 1.4008688926696777, -0.8939194083213806, 0.143364816904068, 0.6633978486061096, -0.16454945504665375, -1.7696893215179443, -0.7785022258758545, -0.1090707778930664, -1.768318772315979, 0.18431878089904785, -0.038046061992645264, 0.5987722873687744, -2.482668161392212, 2.0667901039123535, -0.4246225655078888, 0.40093979239463806, -0.7492915391921997, -0.20647679269313812, 0.6705132126808167, -0.06915006786584854, -0.5687225461006165, 0.304029256105423, 0.30500420928001404, 0.7077562808990479, -1.5280386209487915, -0.9837344288825989, -1.4881030321121216, 1.290035605430603, 0.5892727375030518, -0.21011973917484283, -1.4221267700195312, 2.202336311340332, -0.45623376965522766, -0.2793551981449127, -0.9247642755508423, 0.046450287103652954, -0.695734441280365, -0.25224706530570984, 0.17918308079242706, 0.18471288681030273, -0.052259575575590134, 1.5046478509902954, -1.6795188188552856, -1.9498873949050903, 1.0665253400802612, -0.9367929100990295, -0.3950851559638977, -0.4015066623687744, 1.5718679428100586, -0.8129652142524719, 1.935148000717163, 0.34792444109916687, -0.7210696339607239, 0.9154945015907288, -0.9545994997024536, 0.8514404892921448, -0.10337666422128677, -0.01972282864153385, 1.521897792816162, 0.9511059522628784, 0.5602327585220337, 0.16120074689388275, 0.23305654525756836, -0.2546544671058655, -1.0337185859680176, -0.05290794372558594, 0.6315070986747742, -0.12596087157726288, -0.3467865288257599, 1.1002839803695679, 0.2469073235988617, -0.4400368630886078, 0.23115885257720947, -0.5860926508903503, 1.5832139253616333, -0.21077802777290344, 0.5142389535903931, -1.963979959487915, 0.22342164814472198, 0.3809269666671753, 0.26696279644966125, 0.3477960526943207, -0.29500913619995117, -2.219723701477051, -0.08510144799947739, -0.5204433798789978, 0.8153636455535889, 1.4665452241897583, 0.8655094504356384, 1.4172797203063965, 0.3466571867465973, -0.7238678932189941, 0.8207527995109558, 0.5896734595298767, 0.08896699547767639, -0.4541708528995514, -0.9891730546951294, -0.6542413830757141, -0.6164594888687134, -0.15225850045681, -0.20027683675289154, -0.645802915096283, -0.19139604270458221, -1.1895939111709595, 0.3039153814315796, -1.2810310125350952, -0.7051825523376465, 0.44111689925193787, 0.6545783281326294, -0.8396939039230347, -1.3826943635940552, 0.14989718794822693, 0.7209239602088928, 1.720863699913025, 0.11460497230291367, 0.2463931292295456, -0.3464750051498413, -0.35223841667175293, -1.612608790397644, 0.1565886288881302, 0.13730739057064056, -0.11521975696086884, 0.9785453677177429, -0.7286069989204407, -1.8944693803787231, 0.993161141872406, 0.586096465587616, -0.3079763948917389, -2.6237406730651855, -0.5352174639701843, 0.9612926244735718, 0.002847734373062849, 2.6496596336364746, -0.09918957203626633, 0.20340164005756378, -1.0731065273284912, -2.3717174530029297, -0.5969845652580261, 0.9778738021850586, 0.2408091127872467, 0.5553200840950012, -0.00317813316360116, 0.16124700009822845, 1.0132837295532227, -0.8471469283103943, 0.8372588157653809, 0.6646435856819153, 0.11187087744474411, -0.8970057368278503, -0.1327611654996872, 0.5897473692893982, 0.016344774514436722, -0.9961791634559631, 1.4370828866958618, -0.9530822038650513, 0.31286144256591797, 0.6873693466186523, -0.8006920218467712, 0.06858131289482117, 3.4192051887512207, -2.0777804851531982, 0.27503564953804016, 1.2252904176712036, -0.3318418860435486, -0.8440965414047241, -0.21146251261234283, 1.402860403060913, 0.6570298075675964, -0.6542120575904846, -0.3432224690914154, 0.5609002709388733, 0.6591408252716064, -0.06392697244882584, 0.4173579812049866, 0.10087238997220993, -0.991638720035553, -0.7349326610565186, -0.13570158183574677, -1.741727590560913, -1.0274248123168945, -0.33040058612823486, 1.059554934501648, 0.5822531580924988, 0.7552478909492493, -1.9422037601470947, 0.9018498063087463, -1.9122627973556519, 0.15830405056476593, 0.37071773409843445, -0.7374281287193298, 0.001924613257870078, 0.6901251673698425, -0.30408814549446106, 0.07895946502685547, -1.801175594329834, -1.486077070236206, -0.17556963860988617, 1.0872777700424194, 0.09147192537784576, 0.47330254316329956, 0.033834829926490784, 0.3956688940525055, 0.09891126304864883, -1.1510370969772339, 0.3593858480453491, -0.6962456107139587, -3.386945962905884, -1.7537232637405396, -0.43492868542671204, -0.31783145666122437, 0.4149870276451111, 0.553523063659668, -0.18223275244235992, 0.20781876146793365, 0.3802541196346283, 1.2516448497772217, 0.2669844627380371, -1.3661295175552368, -0.27942776679992676, -0.38390636444091797, -1.178945779800415, -2.890148639678955, 1.2873731851577759, -1.2552070617675781, 2.0349502563476562, 1.7588489055633545, 0.020808443427085876, -0.6529853940010071, -0.23856893181800842, 0.7911509275436401, -0.6263325214385986, 0.7044380903244019, 1.1339932680130005, -1.4968178272247314, -0.40787264704704285, 1.6350377798080444, 1.311497688293457, 0.9347408413887024, -0.6277145147323608, 0.09565069526433945, -1.121069312095642, 1.7337514162063599, -0.9363563656806946, -1.3981796503067017, 0.13379015028476715, 0.300922691822052, -0.40553152561187744, 0.4512315094470978, 0.8141300678253174, 1.356887698173523, 1.1818231344223022, -0.8564108610153198, 0.3576469421386719, -2.2708301544189453, -1.6230576038360596, 1.4089916944503784, 0.8747277855873108, -0.44604313373565674, -2.161815881729126, 0.18933475017547607, 1.2765861749649048, 1.3064534664154053, 1.5761107206344604, -0.6912767291069031, -0.008968882262706757, 1.6366461515426636, -0.5862209796905518, 0.3472331166267395, -0.6120896935462952, -0.27055031061172485, -0.6308576464653015, 2.4328253269195557, 0.45674315094947815, -1.1161271333694458, -0.29424411058425903, 0.6847537755966187, 0.20890995860099792, -0.8051560521125793, 0.46124663949012756, -0.9381768107414246, 2.3722195625305176, 0.4921380579471588, 0.6777032613754272, -0.7053888440132141, -1.6059722900390625, 0.3545433282852173, -1.0978870391845703, 0.49171215295791626, 0.28825515508651733, 2.653679609298706, 0.6889254450798035, -1.076454520225525, 1.3821924924850464, -0.6879011392593384, 0.943274199962616, -0.3420866131782532, -0.3558138310909271, -1.6917699575424194, -0.15432143211364746, -1.016191005706787, 0.15092504024505615, -0.7406970858573914, 2.701627016067505, 0.10990725457668304, -0.1974170207977295, -1.8591614961624146, -0.9505629539489746, -1.0919452905654907, -0.43086084723472595, -0.28510206937789917, 0.36580294370651245, -0.12749230861663818, 0.45932766795158386, -0.5665293335914612, -2.49526309967041, 0.8027134537696838, -0.08115199953317642, 1.8194633722305298, -1.099168300628662, -0.21521717309951782, 0.09353504329919815, -0.04524746164679527, 0.6347360610961914, -0.2094261199235916, 1.4589378833770752, 1.05410897731781, 0.37325650453567505, -0.8762088418006897, -0.06146562844514847, -0.7051975727081299, 0.9384089112281799, 0.04143214970827103, 0.8392611145973206, -0.7587757706642151, 0.2666558027267456, -1.2233805656433105, -0.2655724287033081, -0.5155434012413025, -0.7546696066856384, 0.5633415579795837, -2.5049803256988525, -1.0454379320144653, -1.735522985458374, 2.3890342712402344, -0.060666125267744064, -0.012725270353257656, 0.8028358817100525, -0.5360232591629028, -0.0961398333311081, 1.1101629734039307, 1.3534927368164062, 0.3715018630027771, -1.578687310218811, -0.5994546413421631, 0.862934410572052, -0.5051978826522827, -0.6364738941192627, -1.2072103023529053, 0.9395370483398438, -1.2874311208724976, 0.6029168367385864, 0.8382772207260132, -1.0705299377441406, 0.4062786102294922, -2.6600868701934814, -0.4420774281024933, -0.5230551958084106, -0.4532874524593353, 0.6637082099914551, -0.25776880979537964, -0.4873466193675995, 0.7677730917930603, -0.5632463693618774, -1.5252516269683838, -0.34167617559432983, 1.4406392574310303, 1.3594845533370972, -1.6674375534057617, -0.16912990808486938, -0.7817422151565552, 1.2344332933425903, 0.5640854239463806, 1.5138300657272339, 1.688303828239441, -1.6672353744506836, -1.677297592163086, -0.7858204245567322, 0.563640296459198, 0.036787789314985275, 1.5629583597183228, 1.7724374532699585, -0.3843323588371277, -1.6308022737503052, 0.08651328086853027, 0.7232274413108826, -1.1247917413711548, -1.3904203176498413, -0.5653007626533508, -1.6714541912078857, 2.87153959274292, -0.9580400586128235, 2.9263367652893066, 0.30161234736442566, 0.716629683971405, -1.5489977598190308, -1.4803483486175537, -0.46043166518211365, -0.04424400255084038, 1.1159451007843018, -0.03517281636595726, -1.268766164779663, -0.19762538373470306, 1.1061488389968872, -0.09902627021074295, -1.1970250606536865, 0.2894488573074341, -1.0142861604690552, -0.021423321217298508, -0.4250417649745941, -1.4776300191879272, -1.167375922203064, -0.3980908691883087, 0.8365200161933899, 2.21400785446167, -2.507899284362793, 0.9803306460380554, -0.8643361330032349, 1.133145809173584, -0.3794688880443573, 1.4165252447128296, -0.5520011782646179, 0.4736475944519043, 1.972870111465454, -0.7730768918991089, -0.12516894936561584, -0.55316561460495, 0.6299464702606201, -1.9125547409057617, 0.42274895310401917, 0.3532312512397766, 1.1203627586364746, 0.8376008868217468, 0.43253806233406067, 0.8122631311416626, -1.0929068326950073, 1.072580099105835, 0.9755725264549255, -0.056704841554164886, -0.5600903630256653, -0.13503018021583557, -1.3025546073913574, -0.6584733724594116, 0.3376102149486542, 0.49820321798324585, -0.7375298738479614, 0.18989261984825134, 0.6277474761009216, -0.3755604028701782, -0.760968804359436, -0.2568725049495697, -0.8421806693077087, -1.0010673999786377, -1.0695167779922485, 0.6785886287689209, 0.7725874185562134, 0.40264052152633667, 0.2412022203207016, 2.0336496829986572, 1.696833848953247, -0.286186158657074, -0.0566268116235733, 0.16811169683933258, 0.594578206539154, -0.49326542019844055, -1.3467768430709839, 0.9673371911048889, -0.4183602035045624, 2.5659422874450684, -0.13380150496959686, 0.6012110710144043, -1.1906753778457642, 1.1811403036117554, -0.6409911513328552, 0.8435015082359314, -0.1505829095840454, -1.3154956102371216, 0.11486842483282089, 1.1265169382095337, 0.8242015242576599, 0.38186541199684143, -0.1831386685371399, -1.8890514373779297, -3.2490017414093018, 1.3050512075424194, -0.6700930595397949, -0.8441844582557678, -0.4279966652393341, 0.37176084518432617, 1.3868539333343506, 1.3436874151229858, -0.42856913805007935, -0.047287717461586, -0.3148781359195709, 0.721947193145752, -0.2856461703777313, -1.0954378843307495, -0.03716291859745979, 0.5294620990753174, 0.40369224548339844, 0.43524467945098877, -1.1812009811401367, 0.6371419429779053, -0.41318729519844055, -0.264674574136734, 0.5528413653373718, 0.0024473913945257664, -2.419340133666992, -0.9822428822517395, -0.39227136969566345, -0.311408668756485, 0.8731386065483093, 1.3532551527023315, 0.6645965576171875, -0.7626169323921204, -1.0275208950042725, 0.7647995948791504, 0.3996199369430542, 1.4138545989990234, 0.11666575074195862, 0.22107647359371185, -1.2706928253173828, -0.934856653213501, -0.9938907027244568, -1.6564381122589111, -0.44920164346694946, 0.8163436055183411, 0.19819754362106323, 0.18663369119167328, 0.010778533294796944, -0.7348485589027405, 1.5721243619918823, 0.16363853216171265, 1.1434930562973022, 0.050276029855012894, -0.2287227362394333, 0.30339503288269043, -0.035353370010852814, -0.6353148221969604, 0.7592261433601379, 0.5447120070457458, -1.7795307636260986, -0.25088930130004883, -0.7556019425392151, 0.6875774264335632, -0.32409733533859253, -1.389412522315979, -0.3330942988395691, 0.527197539806366, -0.12546850740909576, 1.4594181776046753, -0.6902050375938416, 1.368750810623169, 0.3363703787326813, 0.520486056804657, -0.48928332328796387, 1.0645899772644043, -1.7543121576309204, -1.4001474380493164, 1.026901364326477, 0.5114902257919312, 2.178400754928589, 0.1635420024394989, -0.05768987163901329, 0.12937936186790466, -0.5299695134162903, 1.152216911315918, 0.28366661071777344, 1.0571966171264648, -0.015100309625267982, -0.4304320812225342, 0.9016740918159485, 1.484103798866272, 0.5860610008239746, -0.44898951053619385, -0.1876598596572876, 0.2503620386123657, -0.3058260679244995, -0.4632359445095062, 1.0428235530853271, -0.09812771528959274, 2.0697555541992188, -1.1452348232269287, -0.0060026682913303375, 0.13524645566940308, 0.4049823582172394, -0.9332419037818909, 1.8341530561447144, -0.4955551326274872, -0.10109204798936844, 1.9268286228179932, -1.5332111120224, 0.001801469479687512, 2.241239070892334, -0.17735762894153595, 0.849141538143158, -1.1294127702713013, -0.9384099245071411, 1.2945247888565063, -0.7179003953933716, -0.14042440056800842, 0.13679447770118713, -1.8431564569473267, 2.262911796569824, -0.22835974395275116, -0.7382813692092896, -1.6074366569519043, 0.704151451587677, 2.2452354431152344, 0.08716558665037155, -0.12488343566656113, -1.098284125328064, 0.29385000467300415, -0.8269894123077393, -0.41497668623924255, 0.5750294923782349, 0.2605806887149811, -1.299988031387329, -0.3841063678264618, 0.1265014261007309, 0.12282751500606537, 1.8508803844451904, -1.113445520401001, 1.2572221755981445, 1.0425684452056885, -1.6070201396942139, -0.14466464519500732, -0.8495645523071289, -0.39589589834213257, -1.372869849205017, -0.2035403996706009, 0.9235909581184387, 0.11132016777992249, 0.17086316645145416, 0.5851283669471741, 0.9404574036598206, 1.6352144479751587, 0.6847352981567383, 1.3620600700378418, -0.4169875383377075, -0.38976988196372986, 1.820708990097046, -0.20563143491744995, -0.5658403635025024, 0.32120272517204285, -0.66241455078125, -0.34938502311706543, 0.3521312475204468, -0.811567485332489, -0.11555250734090805, -0.06452854722738266, -0.17970912158489227, 0.5232206583023071, 0.2264188975095749, -0.306128591299057, -1.1203007698059082, -0.23230765759944916, -0.7114571928977966, -0.678565502166748, -1.1533693075180054, 0.2933367192745209, -0.03327643871307373, 0.3428659737110138, 1.5499142408370972, 1.2896363735198975, -0.9373061060905457, 0.7646278142929077, -1.5077686309814453, 1.0164064168930054, -2.4828686714172363, 0.3777332007884979, -0.4381411373615265, 0.9312198758125305, -1.2637625932693481, 0.923183798789978, 0.8638577461242676, -1.138088345527649, 0.2854284346103668, 1.907396912574768, 0.09234530478715897, -0.42992591857910156, 0.30405840277671814, 0.4752856194972992, -0.2082328349351883, 0.4383839964866638, 0.6726198792457581, -2.288813591003418, 2.2335147857666016, 1.2936204671859741, -1.4723963737487793, -1.2397223711013794, 1.9484285116195679, -0.19394369423389435, -0.9038413166999817, 0.8790102601051331, -0.49807050824165344, -0.4675595462322235, 1.1754224300384521, 0.5863289833068848, 0.6945270895957947, -1.7799032926559448, -0.4840165674686432, 0.021283645182847977, 0.9979339838027954, 1.0508404970169067, -0.5336557626724243, 0.14979155361652374, 0.7484363913536072, -0.24897582828998566, 0.6657993197441101, 0.3858354687690735, 0.9377567172050476, -0.3930363655090332, 0.31797540187835693, 1.5045334100723267, 0.3999682366847992, 0.022049373015761375, 0.5910375118255615, -1.6994047164916992, -1.5843733549118042, 0.4682558476924896, -0.35988548398017883, -0.1737932562828064, -0.7597929239273071, -2.28751277923584, -1.289060354232788, 0.04906830936670303, -3.0790982246398926, -0.6508281230926514, 2.864320755004883, 0.48543792963027954, 1.0876826047897339, 0.16254575550556183, -1.2429723739624023, -0.0497155487537384, 1.0593235492706299, -0.7707913517951965, -1.4823147058486938, 0.21757446229457855, -0.08587148040533066, 0.191848024725914, -2.3140718936920166, -0.38995009660720825, 0.13511085510253906, 0.4356527030467987, 0.4910145103931427, 1.3476585149765015, -0.49133989214897156, 0.7824133634567261, -0.7396373152732849, -0.18082676827907562, -1.5762585401535034, 1.4010570049285889, -1.3930213451385498, 0.31558847427368164, -0.18711717426776886, 0.15216681361198425, -0.14718785881996155, 0.5473012924194336, 0.02734658122062683, 0.4596724212169647, 0.42216265201568604, 1.1167799234390259, -0.35386404395103455, 1.0471237897872925, 0.40994226932525635, 0.2729184031486511, 0.5425226092338562, -0.5835113525390625, 0.5370833277702332, 0.4283870756626129], [-0.8981336355209351, -0.10172683745622635, 0.8360641598701477, -0.16702096164226532, 0.08635728061199188, 0.2675081789493561, 1.0756638050079346, -1.9275593757629395, -0.6712248921394348, -0.6721888780593872, 2.4337007999420166, -0.39303135871887207, 0.37177228927612305, 0.44461289048194885, -0.09005434066057205, -0.9430708289146423, -0.48519784212112427, -0.3136864900588989, 0.19711072742938995, -0.843938410282135, -0.7810764312744141, 2.134737014770508, 1.0055090188980103, -0.0512741282582283, -0.8773811459541321, 0.8887119293212891, 1.0588299036026, 0.523329496383667, -0.6316229104995728, -2.521883249282837, -0.01798519678413868, 0.17093941569328308, 1.4861135482788086, 2.1298325061798096, 0.3629680871963501, -1.505188226699829, 1.1633994579315186, 0.42220064997673035, 1.0596939325332642, 0.9921965003013611, 0.627653181552887, 0.7191666960716248, -0.465772420167923, 0.25869202613830566, -2.587041139602661, 0.20032678544521332, 0.5569155216217041, -1.0334774255752563, 0.1183527335524559, -1.0782361030578613, 0.6005911827087402, 0.6004643440246582, 2.1939306259155273, -0.266761839389801, 0.19770540297031403, 1.1527103185653687, 0.47768500447273254, -0.45306092500686646, 0.0904354378581047, -0.27940312027931213, 0.9859800934791565, 0.08875025063753128, -0.2362828552722931, -0.828220784664154, -0.9570794701576233, -1.7678332328796387, 0.2162596732378006, 0.36423933506011963, 0.29816511273384094, -0.05242704600095749, 1.3839123249053955, -0.03083398938179016, 0.355995774269104, 0.07660814374685287, -0.028951026499271393, -0.32394278049468994, -0.5274069309234619, 0.5025652050971985, 2.1880555152893066, 1.8895671367645264, -0.41976475715637207, -0.7532148957252502, -0.4160962402820587, -0.934567391872406, 1.6575299501419067, 0.6156932711601257, 1.1286582946777344, -0.23404166102409363, 1.5894912481307983, 0.9667181372642517, 0.5609370470046997, 0.6041972041130066, -0.93550705909729, -0.7049564719200134, -1.7029502391815186, 0.8808371424674988, -1.7974507808685303, -1.492478370666504, -0.5975126028060913, -0.17431285977363586, -1.6053096055984497, -0.14908017218112946, -1.4067128896713257, 1.277226209640503, -0.4102914035320282, -1.4252376556396484, -0.3378496468067169, -0.10805664211511612, 0.8679503202438354, 1.345935344696045, 1.2104847431182861, -0.29615697264671326, -0.6297776103019714, -1.3262721300125122, -1.7625596523284912, -0.25541606545448303, 0.7514716386795044, 0.45289888978004456, -0.17043206095695496, -0.38475143909454346, 1.8760747909545898, 0.9048877358436584, -0.166315495967865, 0.16357210278511047, 0.6932308077812195, -0.13239546120166779, -0.026264918968081474, 0.8276199102401733, -0.2822774052619934, -0.09161750972270966, 0.9207720756530762, -1.8817650079727173, 0.6680501103401184, -0.3912237286567688, -0.6064073443412781, -1.1888234615325928, 0.9174601435661316, 0.2961718738079071, -0.3041298985481262, 1.0447521209716797, -0.16161128878593445, 1.2315598726272583, -0.17066636681556702, -1.6412558555603027, -1.6092218160629272, 0.11542940139770508, 0.7943540215492249, 1.300957441329956, 0.7509822845458984, 0.5611032843589783, 0.9227131605148315, 0.4502373933792114, 1.5943472385406494, -0.48141738772392273, -0.30736249685287476, -0.9124184846878052, -0.7831071615219116, -0.43913698196411133, 0.3225490152835846, -0.6145786046981812, -1.9109359979629517, 0.619023323059082, 0.05785535275936127, 1.3106296062469482, 0.6975817680358887, -0.26461732387542725, 1.8180205821990967, -0.05494323745369911, 0.10936113446950912, -1.0597189664840698, -0.6604452133178711, 2.1794593334198, 0.8347898125648499, -0.024092404171824455, -0.9246208071708679, 0.42522433400154114, 1.151939034461975, -1.1117877960205078, 0.9951868653297424, -0.5690943002700806, 0.5069388151168823, -0.3650866746902466, 1.2477867603302002, -1.6241933107376099, -1.173496127128601, -0.43052324652671814, 1.5767139196395874, -0.08907799422740936, -0.8025583028793335, -0.422211229801178, -1.294835090637207, 1.1139287948608398, 1.5226519107818604, -1.1228219270706177, -0.2821955382823944, 0.4525383710861206, -0.8330708742141724, 2.9215590953826904, 1.9001133441925049, -0.9338144659996033, 1.5184924602508545, 0.12380751222372055, 2.3014910221099854, 0.3628650903701782, -0.7645632028579712, -0.7127703428268433, 0.8285801410675049, -0.49281036853790283, -0.6396717429161072, 0.06923675537109375, -1.4344909191131592, -0.17365333437919617, -0.013148115016520023, -0.5938262939453125, 0.7304442524909973, 1.313145399093628, -1.076169490814209, 0.2672930657863617, 0.09180343896150589, 1.687284231185913, 0.5940761566162109, 1.0769392251968384, -1.569568157196045, 1.0536779165267944, -1.0185554027557373, -0.7111165523529053, 0.5296671986579895, -0.2758084833621979, 0.32129883766174316, 0.24050049483776093, 0.5152249336242676, 1.5114398002624512, -0.7326749563217163, 1.4119216203689575, -1.579467535018921, 0.6896395683288574, 0.16801388561725616, 1.857485055923462, -0.8664610981941223, 0.2235657274723053, -0.8025943040847778, -0.08334491401910782, 1.6233707666397095, 2.1596641540527344, 0.7839691638946533, -1.2487363815307617, 0.2235228568315506, 0.0611308254301548, 0.3303990364074707, -0.9239005446434021, -0.9905058145523071, -0.08492227643728256, -0.13584299385547638, -0.28727054595947266, 0.7460851669311523, -0.598482072353363, -0.37608399987220764, 0.25688961148262024, 0.20360970497131348, -0.12364919483661652, 0.24168232083320618, -0.3867771029472351, -2.9238979816436768, -1.9896063804626465, -2.135591745376587, -0.0026875638868659735, 0.4840533435344696, 2.00002121925354, -1.2428666353225708, -0.4830246567726135, -0.8488169312477112, 0.7310729026794434, 0.185268372297287, -0.6112183332443237, 0.7526538968086243, 0.4166650176048279, 0.7484713792800903, 0.127890944480896, -0.7862505912780762, 0.6113478541374207, -0.5729177594184875, 1.3244308233261108, 0.054092928767204285, 0.9339309930801392, 1.0953785181045532, 1.0199356079101562, 1.9267396926879883, -0.9242202043533325, 2.5893313884735107, 0.6946287155151367, -1.5186249017715454, -0.5313185453414917, 0.8422343134880066, 0.038771457970142365, -1.451828956604004, 1.5962063074111938, -0.23103640973567963, 0.7930459976196289, 1.5796979665756226, -0.5562050342559814, -0.2480849176645279, -0.6230560541152954, -1.0779750347137451, -0.31694814562797546, -0.26319923996925354, -1.5749362707138062, 0.05606430396437645, 0.4734249413013458, 0.5349050760269165, 0.23586848378181458, 1.1621146202087402, 0.3090392053127289, -2.086468458175659, 0.1843014508485794, -1.5492459535598755, -0.5742999315261841, -1.673325777053833, 1.0091872215270996, 0.7695342302322388, 0.7150461673736572, 0.7985548377037048, -0.8469216227531433, -0.7670578360557556, 1.1429160833358765, 0.7611967921257019, -1.6403526067733765, -1.9073296785354614, -0.4115615487098694, -0.5563711524009705, 1.34148108959198, 0.8979372382164001, -0.5741187930107117, -0.08570567518472672, -0.9151507616043091, 0.1873237043619156, -1.102504849433899, -0.45454806089401245, 0.7723615169525146, -1.9217623472213745, 1.1932953596115112, 0.9931669235229492, 3.045172691345215, 0.46947771310806274, -0.739863932132721, -1.1853638887405396, -0.5155580639839172, 0.630350649356842, -1.568673849105835, -0.7875100374221802, -1.3655164241790771, -0.3170197010040283, -0.7343451976776123, 1.321361780166626, 0.7453766465187073, -1.4548428058624268, 2.522688865661621, 0.22898392379283905, -1.27163565158844, -1.1443365812301636, -1.6047204732894897, -0.2132897973060608, -0.9653847217559814, 1.2035835981369019, -0.7020730376243591, 1.2436898946762085, -0.4843105971813202, 1.0279252529144287, -0.4801790416240692, 1.7724546194076538, -1.4375025033950806, 0.00872739776968956, 0.3100861608982086, 0.16492938995361328, -0.7351046800613403, 0.10464213043451309, -0.8200350403785706, 1.2009273767471313, 0.1530744433403015, -2.040034294128418, -0.6854659914970398, -0.05150971934199333, 0.45221060514450073, -0.4788399338722229, 0.003763218643143773, -0.385006308555603, 0.1643718034029007, 2.722708225250244, 0.012988617643713951, 2.263876438140869, 2.0270156860351562, -2.7353038787841797, -2.2248735427856445, -0.9031813740730286, -0.04869556054472923, 0.10893595218658447, -0.8438114523887634, 0.18375085294246674, -1.7144420146942139, -0.18615283071994781, -0.5675268173217773, -0.21755698323249817, -0.40285083651542664, -1.1330639123916626, -0.06691277772188187, 0.9014015793800354, 0.1563849002122879, 0.32511886954307556, 1.5495301485061646, -0.8030527234077454, -1.2856683731079102, -1.2908414602279663, -0.318793386220932, 0.9073956608772278, 0.8629937767982483, 1.1164807081222534, 0.7391380667686462, 0.8375260233879089, 0.765942394733429, -0.5579293966293335, -2.0721242427825928, 1.5025850534439087, -2.1548802852630615, -0.36503222584724426, -0.5318712592124939, -0.34330394864082336, 1.3817205429077148, 1.3149112462997437, 0.4702419340610504, -0.8533875346183777, 0.6395918130874634, 1.2148414850234985, 1.0220179557800293, -0.38393768668174744, 1.064945101737976, -1.0420187711715698, -0.12840650975704193, 1.6434600353240967, -1.850295066833496, 0.9005199074745178, -0.3274838924407959, -0.04236087203025818, 0.3132413625717163, -0.08668876439332962, -2.1808881759643555, 0.2191680371761322, 0.11044622212648392, 0.1702686995267868, -0.5167441368103027, 0.6154528260231018, -0.23824277520179749, 3.2014005184173584, 0.14440558850765228, 0.4044872522354126, -0.15856771171092987, -0.8778901696205139, 0.8329457640647888, -0.9485923647880554, -0.9779592752456665, 0.16866490244865417, 1.0156044960021973, -0.5481821894645691, -1.22283935546875, -1.7840280532836914, -0.3890663981437683, -2.301109552383423, -0.8712742924690247, -2.6996641159057617, 0.3132047951221466, 1.2811168432235718, 0.5265485644340515, 0.31175824999809265, -0.4763438403606415, -0.09310091286897659, 1.1041386127471924, 1.1374313831329346, 0.059126414358615875, -1.6616840362548828, -0.4539269804954529, -2.971740484237671, 0.9375216960906982, -0.3096497654914856, 0.6599906086921692, 1.039278507232666, 0.7418889403343201, -0.5149328708648682, 0.032038163393735886, 0.7512388229370117, 1.2672557830810547, 1.1074782609939575, -0.14826452732086182, 1.5541486740112305, -0.08565795421600342, -0.6767908334732056, 0.5036028027534485, -0.8261966109275818, 0.7975966334342957, 0.023580223321914673, -0.3294874429702759, 0.15528884530067444, -0.3143109679222107, -0.8436235189437866, 1.8501744270324707, -1.1027864217758179, -1.112012267112732, -0.14532223343849182, -1.2901064157485962, 0.5115121603012085, 0.9628250002861023, -0.5579121708869934, 1.0501240491867065, -0.10683465749025345, -1.5949947834014893, -1.5228590965270996, -0.1954452097415924, -0.2563454508781433, 0.7089155912399292, -0.07068127393722534, 1.3189681768417358, -0.8146711587905884, 0.905960202217102, 0.019951919093728065, 0.9107235670089722, -0.5014044046401978, -1.1575483083724976, -0.7644791007041931, 0.3426993489265442, 2.4832401275634766, -0.395100861787796, -0.5224803686141968, 0.3581654131412506, -1.010839581489563, 1.3229267597198486, 0.44789886474609375, 0.44657430052757263, 0.3398384153842926, -1.284489393234253, 0.4758158326148987, -1.102283239364624, 0.86817866563797, 0.4999275207519531, -0.6718866229057312, 0.4521472156047821, 0.16169027984142303, -0.4451259970664978, -0.2164796143770218, -0.897412896156311, 1.908600091934204, -0.5399608612060547, -0.3061628043651581, 2.8473496437072754, 0.28837645053863525, -0.2860930860042572, -1.3668705224990845, -0.35414382815361023, 0.9366594552993774, 0.7295539379119873, 0.528971254825592, 1.2356882095336914, -0.0024836831726133823, -0.5041543245315552, -2.5550637245178223, 0.20373405516147614, 0.45982256531715393, 1.900850772857666, 1.8504976034164429, -0.2671913206577301, 0.27774888277053833, -1.3428486585617065, 1.1672732830047607, -0.6190968155860901, -1.3047873973846436, 1.0260576009750366, 1.4453620910644531, 2.028496742248535, 0.7315644025802612, -0.32382237911224365, -0.6936522722244263, -0.6947838664054871, 1.6875317096710205, -1.6753729581832886, -0.30602020025253296, 0.16059494018554688, -0.3588818609714508, 0.48204606771469116, 0.8372769355773926, -1.0258337259292603, -0.26215028762817383, 1.0028280019760132, 0.5459488034248352, 0.3796426057815552, 0.31929367780685425, 1.356797456741333, -1.0981181859970093, -1.3049020767211914, 0.25513380765914917, -1.1207759380340576, 0.5344353318214417, -1.4375965595245361, 0.14322631061077118, -1.7150033712387085, 0.1330486238002777, 0.0004035577003378421, 1.6690599918365479, -0.11008320748806, -0.5468987226486206, -1.1634103059768677, -1.2337710857391357, 0.616450309753418, -0.47210121154785156, 0.3304547667503357, 2.145240306854248, 0.35555097460746765, 0.3947688043117523, 1.458247423171997, 2.9693236351013184, -0.96048504114151, 0.05559403449296951, -0.2074575275182724, -0.3814649283885956, -0.9580170512199402, 0.816895604133606, 0.9371870756149292, 0.7350830435752869, -0.7022111415863037, -0.6719677448272705, -0.0694747343659401, -1.4379576444625854, -0.398427814245224, 0.8043748736381531, -0.7861437201499939, 1.7342921495437622, -0.6375514268875122, 0.3702729344367981, 0.4161168336868286, 0.45355334877967834, -0.3273008167743683, 0.6602258086204529, 2.1048662662506104, 0.5144177079200745, -0.09977409988641739, 0.1893739104270935, 0.6290634870529175, -0.12633074820041656, -1.9988423585891724, 1.8491798639297485, -0.4493279457092285, 1.1908003091812134, -0.5183418393135071, -0.06274086982011795, -0.5314752459526062, -0.516002893447876, -1.2457679510116577, -2.843088388442993, 1.44296133518219, -0.9564047455787659, 0.27012795209884644, 0.8814353942871094, 0.3001568019390106, -1.0285149812698364, -0.7059676647186279, 0.09388308972120285, -1.1670455932617188, 0.19997739791870117, 1.1547369956970215, -0.21705776453018188, 1.048079252243042, -0.7409815788269043, -1.2716456651687622, -0.8226693868637085, 1.2256684303283691, -0.39447474479675293, 0.7998555302619934, -0.4769882261753082, 0.9942122101783752, 0.8662638068199158, 2.219822883605957, -0.8826172351837158, -0.6291830539703369, -0.7756378054618835, -2.172081708908081, 1.6855316162109375, 0.6771495938301086, 0.8187193870544434, 0.0841374322772026, 1.1062631607055664, -0.5907943248748779, 0.8838753700256348, 0.06385102868080139, 0.06127635017037392, 0.030720867216587067, -2.947410821914673, -1.1372463703155518, 0.35212692618370056, 1.1046595573425293, 0.9959772825241089, 1.4462921619415283, -0.9607957005500793, -0.22734665870666504, -0.42336398363113403, 0.23242372274398804, -0.49588853120803833, 0.5950191020965576, 0.5131677985191345, 0.7619520425796509, -0.897855818271637, -1.0206393003463745, -0.09968375414609909, 2.364400625228882, -0.12690292298793793, -1.561738133430481, -0.5765511989593506, -1.1887791156768799, -0.3185335099697113, 1.7138923406600952, -0.7795101404190063, -2.1297478675842285, -0.5413935780525208, 0.06085082143545151, 1.0993335247039795, 0.55824214220047, -1.6683452129364014, 0.2629052698612213, -0.10976953059434891, 0.08031108230352402, -1.3217463493347168, -0.12786005437374115, -0.4244811534881592, -0.8592336773872375, -0.137348473072052, 1.3480660915374756, 1.2522950172424316, 0.18087032437324524, -0.11370296776294708, -1.9473347663879395, 0.7071133255958557, -0.9357266426086426, -0.47507497668266296, 0.7636041045188904, -1.1713618040084839, 0.160713329911232, 0.8691301345825195, 0.36221492290496826, -0.7429503202438354, 0.44868582487106323, 0.6136693358421326, -1.545616626739502, 0.679643988609314, 1.0318920612335205, -1.1684378385543823, -0.1556350439786911, 1.2127357721328735, 1.3075814247131348, 0.7873640060424805, 1.742287278175354, -1.4280133247375488, 0.8168965578079224, -0.6091479063034058, 0.841368556022644, 0.6774126887321472, 0.3543117642402649, -1.3446128368377686, 0.8497360348701477, 0.884632408618927, -1.3570387363433838, 0.8783469796180725, 0.8420748114585876, -1.6861329078674316, -0.6352000832557678, -0.15749263763427734, -0.49927324056625366, -0.8474666476249695, 0.3332803249359131, -0.04961441457271576, -0.23233428597450256, 0.8060829043388367, -0.8990046977996826, 0.6703150868415833, -0.17296147346496582, 0.4885474741458893, -1.1473243236541748, -0.06028246507048607, -1.115687608718872, 1.1858066320419312, 0.48464322090148926, -1.2818914651870728, -0.1430712193250656, -1.2065414190292358, -0.5835245251655579, 1.644370675086975, 0.37844914197921753, -1.5510649681091309, 0.8456193208694458, 1.9860221147537231, -1.3813228607177734, 0.9102342128753662, -0.44538527727127075, 0.5671947002410889, -1.406302809715271, 1.583724856376648, 0.45107796788215637, -0.23344886302947998, -1.15707528591156, 1.407869815826416, -0.6852522492408752, 1.122613787651062, 0.20770879089832306, -0.027839308604598045, -1.2237213850021362, 0.08982635289430618, 1.0514849424362183, 0.7354508638381958, -0.2026526927947998, 0.8061307668685913, -1.6234298944473267, -0.13175861537456512, 0.7223193049430847, 0.8966919183731079, -1.4381879568099976, -0.07160983979701996, -1.5022335052490234, -1.2510451078414917, -0.8549044132232666, -0.27939659357070923, -0.2738952338695526, -1.144782543182373, -1.1750109195709229, 0.5680032968521118, 1.2364842891693115, -1.2889535427093506, 0.9658920168876648, 0.13587352633476257, 1.6260042190551758, 1.329957127571106, -0.6910112500190735, -2.901587963104248, -0.9341531991958618, 0.1684391349554062, -2.353821277618408, 0.5905830264091492, 1.2156739234924316, -1.7312918901443481, 1.5217766761779785, -1.710233449935913, 1.0867425203323364, 0.7212501764297485, -0.4742877185344696, 1.5955733060836792, 0.2740659713745117, 0.1418379694223404, -0.6006083488464355, 0.9751745462417603, 0.10225332528352737, -0.34269627928733826, 0.006179278250783682, 1.4204401969909668, 0.8814910054206848, -1.8838896751403809, -1.8839446306228638, -0.4261603355407715, 0.5944905281066895, 1.4104551076889038, 0.15000320971012115, 1.616099238395691, -1.0960524082183838, 0.04319940134882927, -0.13227151334285736, -0.14897333085536957, 0.078110471367836, -0.007280091289430857, -0.025329014286398888, 0.8394725322723389, 0.07963123917579651, 0.8953936100006104, 0.23092882335186005, 0.7421060800552368, -1.2534199953079224, 1.0602731704711914, -1.698269009590149, -1.2642598152160645, 2.058229446411133, 0.23506982624530792, 1.1313140392303467, 0.5441992878913879, 0.903390645980835, 0.9495633840560913, -1.2555251121520996, 0.34228676557540894, 0.9626280069351196, -0.550934910774231, 0.08639314770698547, -0.1572772115468979, -1.183748483657837, -0.4136858880519867, 0.20385868847370148, 0.05397874116897583, 0.4338359534740448, -1.16915762424469, 0.8119977116584778, 0.2509491443634033, -0.5546038746833801, -1.7579439878463745, 1.8045508861541748, -0.5184427499771118, 0.9455322623252869, -0.12626546621322632, -1.65640389919281, 0.724467933177948, 1.5127812623977661, -0.8521656394004822, -0.4704960286617279, 0.13572636246681213, -0.11941004544496536, -0.6275960206985474, 0.10839210450649261, -1.3951325416564941, 0.6409001350402832, 0.7762588262557983, 1.1896629333496094, -0.20898962020874023, 0.4315149486064911, -0.06134934350848198, -0.7438324093818665, 0.8006742000579834, 0.1162416934967041, -0.9501948952674866, -0.8908687233924866, 1.3950549364089966, -0.5383450388908386, 1.8451365232467651, 1.8965507745742798, 1.1871293783187866, -0.2567943334579468, 1.0866516828536987, -0.7665208578109741, -0.31426727771759033, 0.9512220621109009, -0.18115830421447754, -1.0294766426086426, -1.374070405960083, -0.9830992817878723, 1.4702048301696777, 0.4871155619621277, 2.564439058303833, -0.8761130571365356, -1.5116956233978271, -0.45958927273750305, 0.49586066603660583, 0.8865224719047546, -0.6679242253303528, -0.40013137459754944, 0.3099410831928253, 0.06425705552101135, -1.169987678527832, 0.3297123312950134, -0.6326583027839661, -1.476921796798706, -0.14880916476249695, 0.14913873374462128, 0.9053366780281067, 1.8140084743499756, -1.0903027057647705, 2.5070512294769287, 1.6290152072906494, 1.4033902883529663, -0.9876512289047241, 1.249056100845337, -0.6595092415809631, 0.3363597095012665, -0.38695576786994934, 0.740601658821106, -0.23106355965137482, 0.5494239330291748, 0.38800638914108276, 0.153970867395401, -1.0361601114273071, 0.4363294243812561, -0.3830675184726715, 0.7044079899787903, -1.572109580039978, 1.0389602184295654, -0.835578978061676, 0.6822704672813416, -1.7080373764038086, -0.2809489965438843, -1.562848448753357, 1.2825871706008911, 0.09903509169816971, -0.49002647399902344, -0.2578011751174927, -0.14857915043830872, -0.6675938367843628, -0.7620671987533569, -0.3606138229370117, 0.3181725740432739, -0.5717646479606628, 0.3677539527416229, -1.5501937866210938, -0.9523575305938721, -0.22801189124584198, 1.340728998184204, 0.4628671407699585, 0.5411695837974548, -0.8714219927787781, 0.6376246809959412, -1.1648402214050293, -0.3642030954360962, 1.2891273498535156, 0.31007304787635803, -0.32680168747901917, 0.4395647943019867, 0.5067843198776245, 1.7210118770599365, -0.926337480545044, -0.13508328795433044, -1.0280449390411377, -0.5429971814155579, 0.5702062845230103, -1.327385663986206, 1.4691845178604126, 1.3649903535842896, -0.49490392208099365, -0.5736702680587769, -0.3614402711391449, -0.3404359519481659], [-2.0015370845794678, -1.9839277267456055, -0.37963810563087463, -0.11748876422643661, 0.2565610110759735, -0.8446788787841797, -0.26715192198753357, 1.1836310625076294, 0.4553753137588501, -1.9479436874389648, 0.41745537519454956, 0.18098817765712738, -2.0757765769958496, 0.844778299331665, 1.3085376024246216, -1.6704181432724, 0.3960129916667938, 0.5421838760375977, 0.17066234350204468, -0.0037467656657099724, 2.055665969848633, 1.2559046745300293, 0.7008640170097351, -1.193299412727356, -0.5797328352928162, 0.3617096543312073, 1.0544018745422363, -0.40643957257270813, 0.11141902208328247, 1.7219525575637817, 0.1749068945646286, 1.0336534976959229, -1.1412914991378784, -0.11930923908948898, 0.22427326440811157, 0.5852850675582886, 0.4917181730270386, 0.5438439249992371, 0.7115734815597534, -0.8926607370376587, -0.340311199426651, 0.8771650791168213, 1.345911979675293, -1.7436453104019165, 0.7183434963226318, -0.7544214725494385, 0.826362133026123, -0.18236884474754333, 0.27614641189575195, 1.3106367588043213, -2.3235630989074707, -0.25093525648117065, -0.7585790157318115, -1.0772764682769775, -1.011238932609558, -0.13170474767684937, 1.7437174320220947, 0.7679499387741089, 0.93333500623703, -0.40934258699417114, 0.07913176715373993, -1.18507981300354, -0.8717790246009827, -0.0012429581256583333, 0.8109152317047119, -0.3245090842247009, -0.41433095932006836, 0.16308775544166565, -0.2487616389989853, 0.30636340379714966, -0.7216160893440247, -0.3018498122692108, -1.0018587112426758, 0.18372848629951477, -2.758385181427002, -0.5815333127975464, -0.9858573079109192, -0.7142808437347412, -1.1138654947280884, 1.4508440494537354, 1.6208295822143555, -1.2357652187347412, 0.4312271177768707, -0.0075096976943314075, 0.031593624502420425, 2.0161867141723633, 0.24630235135555267, 1.244891881942749, 0.41017016768455505, 1.3325918912887573, 0.02319991961121559, 0.5384528040885925, -0.4655318260192871, 0.6171810626983643, 0.641478419303894, 0.6729040741920471, -0.59894859790802, 0.06597515195608139, 1.2043267488479614, 1.203506350517273, 0.1784217804670334, 2.5510029792785645, -0.4843868315219879, 0.4496668875217438, 4.017688751220703, 2.279916286468506, -1.5970760583877563, -0.8634445667266846, -0.5324885249137878, 2.4363040924072266, 0.9997127056121826, -0.567828357219696, -0.5576648116111755, 2.195744752883911, -1.663238525390625, 0.7716570496559143, 1.0991746187210083, -1.3058596849441528, 0.41759851574897766, 1.176363468170166, -0.4434451758861542, 1.497247338294983, -1.1672959327697754, -0.6610057353973389, -0.25394406914711, -0.33386319875717163, 1.6276373863220215, -1.4728492498397827, -0.02712130732834339, 1.831573724746704, 2.820124626159668, 0.38698115944862366, -1.168330192565918, -0.7444767951965332, -0.4018367826938629, -1.3174163103103638, -0.3807149827480316, 0.4711962044239044, -0.6683354377746582, 0.8255569934844971, 1.0020524263381958, -0.4300360381603241, 1.770373821258545, 0.4757241904735565, 0.05943942815065384, -0.7338998317718506, -1.2097604274749756, 1.4666718244552612, 0.15662983059883118, -0.27270975708961487, -0.37398865818977356, -1.2616963386535645, 1.4116089344024658, 1.163861870765686, -0.21540167927742004, 0.3800695836544037, 0.5452094674110413, 0.10392901301383972, -0.6823925375938416, 1.5206866264343262, -0.829803466796875, 1.2043973207473755, 0.9601368308067322, 1.6435797214508057, 0.25942960381507874, -1.6362730264663696, -0.27258428931236267, -0.3237568736076355, -0.8408223986625671, -1.4764666557312012, 0.5526300668716431, 0.25500521063804626, -0.0937679186463356, -0.7575230002403259, 2.204848051071167, 1.6596647500991821, -0.7501397728919983, -0.14754319190979004, 0.9695595502853394, -0.1350378394126892, -0.4979155659675598, -1.0192209482192993, -0.4405785799026489, -0.04613799974322319, -2.269259214401245, 1.381249189376831, 0.32854345440864563, 0.9917653203010559, 0.595205545425415, -0.08955713361501694, 1.7651633024215698, -0.4012528359889984, 0.2191641628742218, -0.7703338861465454, -0.9531829357147217, -1.1606286764144897, 2.2676663398742676, -0.32916849851608276, 0.2685413360595703, 0.6837740540504456, -0.8348693251609802, 0.91264808177948, 1.3198169469833374, 0.11378698796033859, -1.208551287651062, -1.8079872131347656, 0.47531846165657043, -0.03176826238632202, -0.018892258405685425, 0.3819127380847931, -0.14626489579677582, -0.16107028722763062, 1.5515464544296265, -0.22047331929206848, 1.0510262250900269, -1.3147221803665161, 2.0180275440216064, 0.15747876465320587, 0.815526008605957, -1.9741307497024536, 2.686102867126465, -1.9494000673294067, 0.8748875856399536, 0.23551103472709656, -1.7358105182647705, 0.6380824446678162, -0.5256907939910889, 1.0214388370513916, -0.27025434374809265, 2.4045662879943848, 0.09558488428592682, -0.7459270358085632, -0.9700578451156616, -1.8676766157150269, 0.5334227681159973, 0.4209943115711212, 1.3838181495666504, -0.7927778959274292, -2.261786460876465, -1.3685247898101807, 0.7080910205841064, 0.4365287125110626, 0.33307117223739624, -0.5977675318717957, -0.21289631724357605, 1.043893575668335, 1.099535346031189, -0.4325812757015228, 0.6413442492485046, 2.332505226135254, 1.7997814416885376, 1.0408306121826172, 0.5520092844963074, 0.015711279585957527, -1.8813749551773071, 0.1837085634469986, 1.338392734527588, 0.3283068537712097, -0.1399136483669281, -0.720680832862854, -0.9272171258926392, -1.1687932014465332, 1.6942564249038696, 0.19595502316951752, 0.557157576084137, 0.2144334763288498, -0.09249895066022873, -1.4296609163284302, -0.2447776049375534, -1.5265288352966309, 0.1756104826927185, -0.8390865325927734, -1.2808417081832886, -0.2997596263885498, 1.1900869607925415, -0.2742031216621399, 0.8204094171524048, 1.4666614532470703, -1.2765225172042847, 1.2828452587127686, -1.2645282745361328, -1.692330241203308, 1.2557846307754517, 0.5181198120117188, -1.2116482257843018, 0.5503042340278625, -1.0797278881072998, 0.252191424369812, -0.4486318528652191, -0.1711757779121399, 0.5938668847084045, 1.159643292427063, -0.3274793028831482, -0.33328497409820557, -0.3521232604980469, -0.008230089209973812, -1.6251347064971924, 0.8470600247383118, 1.946235179901123, -0.9625425934791565, 0.40240558981895447, 1.3010599613189697, 0.7615386247634888, 0.24459373950958252, 1.377426028251648, 0.46390005946159363, -1.0208526849746704, 1.3902192115783691, -0.44875961542129517, 0.5845876336097717, -0.9155453443527222, -0.04925836995244026, 0.6262003779411316, -0.3385104537010193, 0.21146173775196075, 0.35263171792030334, -0.9869022965431213, -0.46772339940071106, -1.7007240056991577, -0.7978461384773254, 0.3419525623321533, -0.8915248513221741, -1.2338131666183472, -0.9123350381851196, -0.9821993708610535, -0.07393492758274078, -0.6510483622550964, -0.6450525522232056, -0.7755183577537537, 0.5408225655555725, 0.3912630081176758, -1.5402315855026245, -0.045830994844436646, -0.5504498481750488, 1.1216915845870972, -1.3294597864151, 0.24436156451702118, 0.48000243306159973, 0.770071804523468, -0.05320721119642258, -0.6137353181838989, -0.0605236291885376, 0.8469046354293823, 0.4105808436870575, 2.399087905883789, -0.09861660748720169, -0.709924042224884, 0.6545712351799011, 0.9091367125511169, -0.20452041923999786, 0.37983912229537964, 0.48790255188941956, 0.518873393535614, -0.6571666598320007, -0.9639293551445007, 0.5785881280899048, -1.286312460899353, -1.0584636926651, 0.4864135980606079, -0.596282422542572, -0.47963300347328186, 0.2493743896484375, -0.426898717880249, -1.015533685684204, -0.5523467063903809, 1.047988772392273, 1.0703036785125732, -0.288704514503479, -1.7170456647872925, 0.2513267695903778, -1.2678616046905518, -2.3602547645568848, -0.5311280488967896, 0.5000898838043213, 1.7735153436660767, 0.09674398601055145, -0.894487738609314, 0.673714280128479, 0.9334401488304138, 0.6112133860588074, 1.064199447631836, -0.16808056831359863, 0.42959752678871155, -3.0499684810638428, 0.2675265669822693, 0.005510583054274321, -0.8920537829399109, 1.0334588289260864, -0.19861380755901337, 0.29999738931655884, -0.5861461162567139, -0.4315640330314636, 0.40870070457458496, 1.1445987224578857, 1.4487959146499634, 0.08317240327596664, 0.6402865052223206, -1.1611891984939575, 0.2177603840827942, 0.19308513402938843, 0.3299221992492676, 0.9246285557746887, 1.464976191520691, 0.0031266475562006235, -0.013844267465174198, 0.01815144158899784, -1.9248346090316772, -1.6314600706100464, 0.18307438492774963, -0.5639863014221191, 0.6824188828468323, 1.5198338031768799, -1.158316969871521, 1.355430245399475, -0.2119690179824829, 0.27021896839141846, 2.961364269256592, -0.44970014691352844, 0.2961099147796631, 1.3324429988861084, 0.979019820690155, -0.012633676640689373, 0.2754712998867035, -0.18018345534801483, -0.33648690581321716, -1.2325934171676636, -0.11001521348953247, 0.8641163110733032, 0.6792581081390381, 0.7594257593154907, 0.9243232011795044, 1.4345437288284302, 0.5611041188240051, -1.8587093353271484, -0.5813080072402954, 0.8043229579925537, -0.27981892228126526, -0.03851550072431564, 1.8446617126464844, 0.3962567448616028, -0.12578094005584717, 1.6157417297363281, -0.5981225371360779, 0.8799290657043457, -0.16030526161193848, 0.9466925263404846, -2.0293493270874023, 0.8891383409500122, -0.06352728605270386, -1.2999390363693237, -1.2453694343566895, 0.742647111415863, -0.3423612117767334, 1.6027263402938843, -0.7952787280082703, 1.183505654335022, -0.5133020877838135, 0.6576981544494629, 1.141465663909912, 0.76796555519104, -0.7557426691055298, -1.7437491416931152, -0.7468724250793457, -0.12079575657844543, -0.5200991034507751, 0.18191605806350708, -0.7293943762779236, -0.8151201009750366, -0.13454300165176392, 2.981985330581665, -0.5793508291244507, 1.1163098812103271, 0.49853280186653137, 0.7050585150718689, -0.7706252932548523, -1.294438362121582, -2.4792633056640625, 0.8370119333267212, 0.1529129445552826, -0.3485816419124603, 0.9903320670127869, 1.0079888105392456, 0.3453708291053772, -0.37155699729919434, 0.25982600450515747, 1.0950533151626587, -0.5856671929359436, 0.8579124808311462, -0.1189548596739769, 0.9118715524673462, -1.5941416025161743, -0.5260306596755981, -0.39681896567344666, -0.381783127784729, 1.1186641454696655, -0.5523127913475037, 0.9794551134109497, -1.3290197849273682, 1.1043263673782349, 0.14385229349136353, 1.6757233142852783, -0.7969699501991272, 0.3095807135105133, -0.875393271446228, -2.0698070526123047, 0.5644558668136597, -1.2081433534622192, 0.3586265742778778, -0.90310138463974, 0.7631260752677917, 0.1561688929796219, -0.17476961016654968, -0.17543117702007294, 0.5967353582382202, -0.2970936894416809, 0.49694493412971497, -0.6424103379249573, -1.2181572914123535, 1.2168794870376587, 0.5515931844711304, -0.6663058400154114, 0.27893298864364624, 1.1625447273254395, 0.36312687397003174, -0.39740583300590515, 0.397377073764801, -0.8797183632850647, -0.5936309695243835, -1.2507017850875854, -0.8375825881958008, 0.34047186374664307, -0.10330457985401154, 1.428708553314209, 0.5894157290458679, -0.8448276519775391, 1.8962135314941406, 0.42090317606925964, -0.7777506113052368, 0.5205961465835571, 0.43652617931365967, 0.20344571769237518, -0.005251878406852484, 0.7103040814399719, 0.05914779752492905, -1.6676989793777466, 0.42802509665489197, 1.6851972341537476, -0.5972215533256531, -0.1783563792705536, 0.591528058052063, -2.2610437870025635, 0.21646885573863983, 0.29378461837768555, 0.2141418159008026, 0.9353845715522766, 0.6920363903045654, 0.9813737869262695, 0.1330810785293579, -1.7454743385314941, 1.0197651386260986, 0.9383440017700195, 1.2463412284851074, -1.0584335327148438, -1.0857454538345337, 0.3147014081478119, 1.3134939670562744, 0.2901250123977661, 1.5202666521072388, -0.44034168124198914, -0.6950563788414001, 1.0352365970611572, -0.0906037837266922, -0.5681897401809692, -1.0842052698135376, 0.7553306818008423, 0.009451269172132015, -0.1484188288450241, 1.2499361038208008, 0.1390533745288849, 0.07180063426494598, 0.0634155422449112, 0.6664870977401733, -0.8598077893257141, -0.27084681391716003, -0.44043654203414917, -0.43443143367767334, -0.5275261998176575, 1.1903104782104492, 0.6211808323860168, 0.5316082239151001, 0.21734793484210968, 0.8582640886306763, 0.7909777164459229, 1.67195725440979, -0.20023564994335175, -0.25561174750328064, 1.7091639041900635, 0.6323824524879456, -0.6034919023513794, -1.0405619144439697, -1.0500414371490479, -0.48551714420318604, -0.5159512758255005, 0.04711902141571045, -1.223466157913208, 1.321814775466919, 0.39603230357170105, 0.970209538936615, -1.1440207958221436, 0.8168001174926758, 0.8907248973846436, -0.023111265152692795, 0.29189303517341614, -0.5591808557510376, -0.594597578048706, -0.009573057293891907, -0.09802362322807312, 1.5466927289962769, -0.9124841094017029, 0.0016592127503827214, -1.0371173620224, -0.7659435272216797, 2.0226969718933105, -0.2344381958246231, -1.0899146795272827, 2.0833470821380615, 0.7541399002075195, 0.20917610824108124, 0.7508224248886108, 0.81111741065979, -0.8823765516281128, 0.2580889165401459, 1.3104499578475952, -0.8112143874168396, -0.6107844710350037, -0.5179205536842346, 1.4359544515609741, -0.5286604166030884, 0.9255961775779724, -0.6119551658630371, 0.6955808401107788, -0.6302029490470886, -1.3732757568359375, -1.1116009950637817, 0.48903676867485046, -0.030647825449705124, 0.2839113771915436, 0.9468103051185608, -1.1746000051498413, -0.06523264199495316, -0.986184298992157, 0.618780791759491, 0.06268113106489182, -1.4907351732254028, 1.7195087671279907, -0.20238997042179108, 1.201002836227417, 1.422590732574463, -1.1305726766586304, -0.20093262195587158, 1.1976053714752197, -0.5306472182273865, -1.2556036710739136, 0.7892663478851318, 1.7955831289291382, -0.3248526155948639, -0.023612098768353462, -1.819038987159729, -0.9031991362571716, -0.26371538639068604, 0.6146361231803894, 0.04330793768167496, 1.3696855306625366, -0.22880645096302032, 0.08259520679712296, 0.5587708950042725, -0.8540427684783936, -2.171663284301758, 0.8297277092933655, -1.2640509605407715, -0.25827959179878235, -2.428903579711914, -0.40382421016693115, 0.9944725036621094, 0.8278098106384277, 1.2711620330810547, 0.6246832013130188, 0.13006691634655, -1.0696436166763306, -0.46817976236343384, 0.34407925605773926, 0.2496493011713028, -1.053259253501892, 1.03550386428833, 0.390837162733078, -0.12313856184482574, 0.11281359195709229, -0.7747038006782532, -1.6024035215377808, 0.7623357176780701, -1.2488553524017334, -0.8044790625572205, -0.11147771030664444, 0.9851837754249573, -0.1755339652299881, -0.35773029923439026, 0.767098069190979, 0.5653156638145447, 1.7509268522262573, -0.8401269316673279, 0.1611991971731186, -1.037458062171936, 0.5729867815971375, -1.4741650819778442, 1.101500153541565, 0.243545800447464, -1.2624990940093994, -0.2516010105609894, -1.6805176734924316, -0.12854184210300446, 0.42608317732810974, -0.015877781435847282, -0.40493106842041016, -0.006919799372553825, 0.05898848548531532, 1.414305567741394, -2.395685911178589, -0.2910195291042328, 0.18510188162326813, 0.3305928409099579, -1.0432922840118408, 0.47029924392700195, 0.21362942457199097, 1.5773192644119263, -0.20179681479930878, -0.8606876730918884, -0.41371676325798035, -0.12700796127319336, -0.09720869362354279, 0.1921423226594925, 0.4493979811668396, 2.1512906551361084, -0.8967392444610596, -0.37201133370399475, 1.4262909889221191, -1.6601158380508423, -0.17734801769256592, 0.47818246483802795, 1.1052652597427368, -1.1838113069534302, 1.1128301620483398, -0.29226166009902954, -2.505948781967163, -0.3079375922679901, -0.07952690869569778, -1.1380563974380493, -0.09788443893194199, -1.059463381767273, -0.5218823552131653, -1.7465426921844482, 0.0992831364274025, 0.16049623489379883, 1.4237604141235352, 0.42425715923309326, 0.021362721920013428, 0.49195596575737, -0.15851818025112152, 1.4942562580108643, 0.2313399761915207, 0.3194638192653656, -1.0538426637649536, 1.1460241079330444, 0.43522515892982483, 0.9383676648139954, -0.7898203730583191, -0.9595587849617004, -0.7428262829780579, -0.2824245095252991, -1.13021981716156, -0.7154145240783691, -0.05252490192651749, 0.5496116876602173, 0.74590665102005, -0.6578036546707153, -0.8815447688102722, -0.960877001285553, -0.04420487582683563, -0.21966993808746338, -0.354145884513855, -0.566999614238739, -1.1584714651107788, -2.3464574813842773, 1.1437864303588867, -0.5475146174430847, 1.7141331434249878, -1.0412956476211548, 2.1008975505828857, -0.3934268355369568, -1.553855061531067, -0.9806636571884155, 0.11706526577472687, -0.420668363571167, -0.6287640333175659, -1.4598323106765747, 1.3700217008590698, 1.9472622871398926, -0.5163631439208984, -1.097882866859436, 0.6374371647834778, -0.9586969017982483, 1.971100091934204, -0.6469628810882568, -0.02016279846429825, 0.5245385766029358, -1.4225189685821533, -0.3509505093097687, -1.3925409317016602, 0.46031829714775085, -0.158128023147583, -0.8486855626106262, 0.9868643283843994, -0.2526884078979492, 0.4578786790370941, 1.404247522354126, 1.2022134065628052, 0.9959473609924316, 0.009083237498998642, 1.2171884775161743, 0.30809664726257324, 2.2159056663513184, -0.07579247653484344, -0.19317889213562012, -0.6540142893791199, 1.3712279796600342, 1.052720069885254, -0.9763481020927429, 0.17052115499973297, 0.8234102129936218, -0.5105026364326477, 0.6098237633705139, -0.8236002922058105, -0.0031475978903472424, -1.593814492225647, 1.4176454544067383, 0.5333178639411926, 1.273341178894043, -1.228994607925415, -1.2320880889892578, -0.1860986351966858, 1.0774705410003662, 0.04291165992617607, 0.42618536949157715, 0.3916717767715454, -0.5347490906715393, -1.468405842781067, -1.540427327156067, -0.22779132425785065, 1.2068175077438354, 0.8882411122322083, 0.5008754134178162, -2.5432751178741455, 0.25581637024879456, -0.23303861916065216, 0.6379812359809875, 0.8699495196342468, 1.0311203002929688, -0.20357748866081238, 0.7418272495269775, 0.6052927374839783, -0.5385260581970215, 0.9342650771141052, 3.37996768951416, 0.2655116319656372, 0.7076976299285889, -0.8905384540557861, 0.3574431240558624, -0.7851153612136841, 0.04160595312714577, -1.7374578714370728, -0.370113343000412, -0.27265214920043945, 0.7684869170188904, 1.0129008293151855, -0.9747335910797119, 0.39821746945381165, -0.17335812747478485, -0.1279200166463852, 0.6045910120010376, 1.1644015312194824, -0.34717094898223877, -0.731381893157959, 0.9338694214820862, 0.1065274029970169, 0.7665466666221619, -0.4393669664859772, 0.8196079134941101, -1.0116714239120483, -0.1647586226463318, 1.204314947128296, 1.316612720489502, 0.1624782681465149, 0.20212917029857635, -0.5820523500442505, 0.6991404294967651, 0.8656399250030518, 1.6222947835922241, -0.1846495121717453, 0.0009525644709356129, -1.13285493850708, 1.3183549642562866, -0.29249897599220276, -2.5009658336639404, -0.4237690269947052, -0.9559023976325989, -0.7123628854751587, 0.15860311686992645, -0.27555882930755615, 0.40693745017051697, -0.7231460809707642, 2.4059064388275146, -0.8101348876953125, 0.5798779129981995, 0.6394458413124084, -0.2116789072751999, -0.25283902883529663, -1.261160969734192, -2.031243324279785, -0.4483151435852051, 0.31018149852752686, 0.5170327425003052, -0.35448670387268066, -0.28053489327430725, -0.33655253052711487, -1.6905499696731567, -1.3209130764007568, -0.5289313793182373, -0.8433288335800171, -0.5477144718170166, -0.7636846899986267, 1.7430506944656372, -0.49945124983787537, -1.008467435836792, -1.9818145036697388, -1.4288133382797241, -0.2503769099712372, 0.8087121844291687, 0.23363040387630463, 0.08068808168172836, 1.0478695631027222, 0.05029672011733055, 0.7894879579544067, 0.7240856885910034, 0.7705701589584351, 1.573393702507019, -0.9735510349273682, 0.0193637702614069, -1.4863115549087524, 1.1494382619857788, 0.07644181698560715, -0.8820959329605103, -0.22453486919403076, 0.6370915174484253, -1.7872514724731445, 0.06196482479572296, 0.3483119308948517, -0.1913740038871765, 0.07641743868589401, 2.2164506912231445, 1.8758922815322876, -1.52970552444458, -1.0121705532073975, 0.25484955310821533, 0.24541880190372467, -0.6525784730911255, 0.9707019329071045, 2.086559295654297, -0.20988798141479492, -0.38479578495025635, -0.08466191589832306, 0.2155902087688446, -0.9214094281196594, 0.1904371678829193, -0.3231014609336853, -2.811241388320923, 1.1981420516967773, -0.19679409265518188, -0.9303330183029175, -1.1512104272842407, 2.1507153511047363, -0.32641786336898804, 0.7311441898345947, -0.8388566970825195, 0.6378812193870544, -0.13300299644470215, -1.1515077352523804, -0.5983567833900452, 0.5690656900405884, 0.024622704833745956, -1.724381446838379, -0.07350548356771469, 0.9592088460922241, -1.9279428720474243, -1.0848718881607056, -0.25681987404823303, 1.8910655975341797, -0.21710996329784393, 0.05076222866773605, -1.3272817134857178, -0.11147487163543701, 0.5670475959777832, 0.3271312713623047, 0.6670029759407043, 0.8788876533508301, -0.5352644324302673, -0.613478422164917, -1.3363913297653198, 0.17712299525737762, -0.7123116850852966, 0.7023333311080933, 0.694854736328125, -0.8789767622947693, -0.48951125144958496, 0.22412244975566864, 0.03481508046388626], [0.9666871428489685, -0.8328168392181396, -0.7371434569358826, -0.06214191019535065, 0.10888894647359848, -0.9229804277420044, 0.35813069343566895, 0.746816098690033, 1.1612809896469116, 0.04941897839307785, -0.8133966326713562, -1.4175513982772827, -0.3081696629524231, -1.068331241607666, 1.6410843133926392, 1.113317608833313, -1.3431296348571777, -0.6207298040390015, 1.8553816080093384, -0.04789543151855469, 0.29148587584495544, -0.5163323283195496, 0.42968496680259705, 0.5829657912254333, 0.643936276435852, 0.01307072676718235, -0.4676452875137329, 0.33947625756263733, 1.5252708196640015, -0.32162144780158997, 1.9254993200302124, -0.42537620663642883, -0.4308965802192688, -0.7014446258544922, -2.2506802082061768, -1.2435003519058228, -0.13446955382823944, -2.7164413928985596, 0.11927928030490875, -0.7197030782699585, 3.4215424060821533, -0.6998748183250427, 1.6625486612319946, -1.6890461444854736, -0.6271410584449768, -0.37891921401023865, -1.5111117362976074, 0.1878279149532318, -2.215825080871582, 0.9071606397628784, 0.7531596422195435, 1.4582788944244385, 1.7513350248336792, 0.8142279982566833, -0.19603820145130157, -0.734687089920044, -1.209356665611267, -0.7781192660331726, 0.17008928954601288, -0.3782027065753937, -0.9563004970550537, -0.916806161403656, -0.23436376452445984, 0.8011614084243774, 0.05438164621591568, -1.2630423307418823, -0.5503709316253662, 0.3252381384372711, 0.9645323157310486, -0.7908445000648499, 0.10159789025783539, -0.08810502290725708, -2.5183908939361572, -0.8679891228675842, 1.0559440851211548, 1.6105831861495972, 0.9863892197608948, -0.13557767868041992, 0.44389522075653076, -0.5001229643821716, -1.0629749298095703, 0.0601506233215332, -1.2970781326293945, -0.04117365553975105, 1.4387391805648804, 0.38884690403938293, -1.6236776113510132, 0.030342120677232742, 0.5267159938812256, 1.3245656490325928, -0.7187996506690979, 0.6757938265800476, -0.3082795739173889, 1.2273802757263184, -0.7071681022644043, -0.9951695203781128, -0.02499212697148323, 0.5812782049179077, -0.4765545725822449, -0.622288167476654, 1.7876337766647339, -1.5054610967636108, -0.22582171857357025, -0.17814484238624573, 0.9472385048866272, -2.6497018337249756, -1.9223968982696533, 1.5375109910964966, -0.41154974699020386, -0.8600125908851624, -0.36958226561546326, -0.31340333819389343, -0.3618682324886322, 0.27474233508110046, -0.3050219416618347, 1.3263736963272095, 0.3389221131801605, -0.9108618497848511, -0.9252293705940247, 0.6318579316139221, 0.939861536026001, 0.3434351086616516, -1.0959261655807495, 1.0455421209335327, 1.6944494247436523, -1.0222574472427368, -1.6769659519195557, -0.5537105798721313, -0.9946841597557068, -0.8569914698600769, -0.3349343538284302, 0.7691333889961243, -0.14158102869987488, 1.979500651359558, 1.718284249305725, -0.09770560264587402, 1.4702688455581665, -0.6354052424430847, 0.2953947186470032, -1.0518673658370972, 0.32041406631469727, -1.3888881206512451, 0.4671976566314697, -2.1358346939086914, -1.0949314832687378, 0.473916232585907, -2.6099960803985596, -0.93104487657547, 1.6426867246627808, 1.4141039848327637, 0.40048688650131226, 1.3702958822250366, 0.9059920310974121, -1.0197327136993408, 0.6584826707839966, 0.3707297146320343, -0.3647429943084717, 1.061884880065918, -0.3751911520957947, 1.8506464958190918, -0.16432708501815796, -0.03669624775648117, 0.6652675271034241, -1.3223189115524292, -1.5407462120056152, -1.0465083122253418, 1.8029823303222656, 0.7027193307876587, 0.803672730922699, 1.9962595701217651, 1.2604966163635254, 0.30531272292137146, -0.3967615067958832, 0.2056647539138794, -1.2739624977111816, -0.08010300248861313, 0.3258751928806305, -2.3096227645874023, -0.018484828993678093, -0.513133704662323, -0.3909127116203308, 0.4532073438167572, -0.38581976294517517, 0.47567984461784363, 0.24868033826351166, -0.4254685044288635, 0.8646946549415588, -1.0761613845825195, 0.3895782232284546, 1.09766685962677, 1.2697392702102661, 0.16997890174388885, -0.27445822954177856, -0.7587193846702576, 0.7155218720436096, 1.3070452213287354, -1.492353916168213, -1.1258600950241089, 1.32307767868042, 0.7570477724075317, -0.14560729265213013, -0.920434832572937, -0.997883141040802, -0.49787694215774536, -0.30007582902908325, -0.5931790471076965, 0.950672447681427, -1.178453803062439, -0.8041301965713501, -0.32652413845062256, 1.2236952781677246, 1.5189841985702515, -0.21245929598808289, 0.875588059425354, -0.7262061238288879, 0.9762948751449585, 0.6018532514572144, 0.4440128803253174, -0.8846741318702698, 0.41241443157196045, -0.5545080304145813, 0.9807657599449158, 1.497751235961914, -0.49747398495674133, -0.5244489908218384, 0.694815456867218, -1.4813220500946045, 0.7773665189743042, -0.3678622245788574, -1.6791422367095947, -1.8772536516189575, 0.4490915536880493, 1.532287836074829, 1.2071495056152344, 0.961485743522644, -1.0840342044830322, 1.7816743850708008, 2.0180766582489014, 1.9342689514160156, -0.2262515276670456, 0.27666550874710083, 0.856956958770752, -1.092400312423706, -0.9797935485839844, -1.4243186712265015, -0.5413393974304199, 0.0644870176911354, -0.641622006893158, 2.4818718433380127, -0.1373637169599533, -0.1751745492219925, -0.1933913230895996, -0.40397587418556213, 1.0028480291366577, -1.3225213289260864, -0.42750316858291626, -0.6242955923080444, -1.0391075611114502, 0.4220018982887268, 0.2701834738254547, 0.3366573750972748, -1.5980232954025269, 1.368902564048767, 0.10364045202732086, -1.2670550346374512, -0.6024194955825806, -0.9740477204322815, -0.011584066785871983, -0.2277757078409195, 0.40874165296554565, 1.329184889793396, -0.24177907407283783, 1.0777850151062012, -1.5979825258255005, 0.8143901824951172, -1.3433886766433716, -0.2248801589012146, 1.2633657455444336, -0.2919325828552246, 1.0523508787155151, 3.1471261978149414, -1.4504432678222656, 1.4059016704559326, -0.34491419792175293, -0.818445086479187, 1.5264489650726318, 0.8795697093009949, -0.7827355265617371, -0.6372645497322083, 1.3552944660186768, 0.4703110158443451, 0.3183366358280182, 0.7663623094558716, -0.7876838445663452, -0.4524880647659302, -1.5003951787948608, 0.8751641511917114, -1.6462712287902832, -0.14802274107933044, -0.03819736838340759, 1.1270946264266968, 0.7705857753753662, 0.6125260591506958, 0.4133983850479126, 0.9261341691017151, 0.5837522149085999, -1.4060736894607544, 0.13027188181877136, 0.001917475601658225, 0.4863017201423645, -0.6509460806846619, 0.40583109855651855, 0.39316868782043457, 0.5471848249435425, -0.07199788093566895, -0.35956865549087524, 0.024870790541172028, 1.0177277326583862, -1.5503369569778442, -0.6003087759017944, 1.0992661714553833, -1.019376516342163, 0.6267867684364319, -0.33529406785964966, -0.06634457409381866, -0.3483424186706543, 1.1468220949172974, 0.1407657116651535, 0.8225165009498596, -0.2302320897579193, 0.11806359142065048, 0.4837673306465149, -1.9983155727386475, 1.8538285493850708, 0.08244078606367111, 0.5298425555229187, -0.5076157450675964, -0.6959671974182129, -1.1715911626815796, -1.5555775165557861, 0.7370004057884216, -0.9398083090782166, 1.0955497026443481, 1.5632253885269165, 0.821712851524353, -0.2948369085788727, -1.163692831993103, -0.29804450273513794, 1.195422649383545, -0.06868597120046616, 0.7769966125488281, 1.3735339641571045, -0.9890369176864624, -1.6407206058502197, 0.3266717493534088, 0.03220876678824425, -0.1882314532995224, 0.3252854347229004, -0.01684746891260147, -0.7331846952438354, 0.6060808897018433, -1.979576587677002, 0.4496774971485138, -1.4735594987869263, -2.7313826084136963, -1.0612061023712158, 0.2697787880897522, 1.153287649154663, 0.08871366083621979, -1.0886260271072388, 1.2151659727096558, 0.4525814354419708, -1.384761929512024, 2.6408543586730957, -0.2537142336368561, 0.7709306478500366, -0.6962723135948181, -0.12425880879163742, -0.05684360861778259, -0.5809565782546997, -0.6742922067642212, -1.6753062009811401, 0.3546464443206787, 0.18681718409061432, -0.779317319393158, 1.333215594291687, -0.60158771276474, 0.5463850498199463, -1.6383945941925049, 0.7178322672843933, -0.27285802364349365, 1.6487557888031006, 0.3858828842639923, -1.1678967475891113, 1.768452525138855, 1.002653956413269, -0.14241594076156616, 0.4908703863620758, 0.4905082881450653, 1.4754217863082886, 0.17968855798244476, 0.5869438052177429, 1.8293678760528564, -0.5327621102333069, -0.2737095355987549, 0.07720108330249786, -0.15965807437896729, 0.24378585815429688, -0.9043079018592834, -0.203505277633667, -1.272934913635254, -1.849120020866394, 1.5209554433822632, 1.3859293460845947, 0.03127402812242508, -0.17951178550720215, 0.3293922543525696, -0.29778724908828735, 0.23332183063030243, 0.9975195527076721, -0.5921023488044739, -1.2878206968307495, -0.544545590877533, 1.8028672933578491, 1.0694096088409424, -2.469481945037842, 0.5640748739242554, -0.4983951151371002, -0.7234994769096375, -2.061509132385254, 2.0559327602386475, 2.398745536804199, 0.10456789284944534, 0.5453542470932007, -1.7261124849319458, -0.22996307909488678, -1.4720580577850342, 0.38002869486808777, 1.1336537599563599, 2.5744595527648926, 2.2409255504608154, 0.4872783422470093, -0.06756225228309631, -1.303572416305542, 1.4010429382324219, -0.7376788854598999, -0.5748277306556702, -0.035074472427368164, -0.10031192749738693, -1.0822826623916626, -0.43829846382141113, 0.16377542912960052, -0.13779619336128235, 2.3416695594787598, -0.6742334365844727, 0.9635289907455444, 0.6510787606239319, 1.3600562810897827, -0.9610262513160706, -0.8728835582733154, 1.9590628147125244, 0.44147810339927673, -0.15437108278274536, -1.6648966073989868, 1.7785567045211792, 1.0044633150100708, 0.7647032141685486, -0.3819887042045593, -0.6396639347076416, 0.5056490302085876, 0.015655117109417915, -1.0672286748886108, 1.6590315103530884, 0.6176277995109558, 1.5980802774429321, -0.09184009581804276, -0.7652900218963623, 2.476074457168579, -0.24995504319667816, -0.2911248207092285, 0.578952431678772, 0.7956016063690186, 0.8179543614387512, -1.2784332036972046, -0.5209867358207703, -0.31006720662117004, -0.5473872423171997, -0.21037189662456512, -0.2737991213798523, -0.11638765782117844, -0.026726175099611282, -1.309949517250061, 1.1740562915802002, 2.1654391288757324, 1.2650777101516724, -0.7993040084838867, 1.4889075756072998, 0.8527127504348755, -0.2738223969936371, 2.2512903213500977, 0.8815802931785583, -0.7354878783226013, -0.735873818397522, 0.38563743233680725, -1.5094430446624756, -0.42722293734550476, 0.0874912440776825, -0.14394372701644897, -0.42828303575515747, -1.571434497833252, -1.0649150609970093, 0.04749290645122528, -1.015867829322815, -0.9528002142906189, -0.6933980584144592, -1.6414790153503418, -0.08990627527236938, 0.021655848249793053, -0.9620136022567749, -0.15850834548473358, 0.3712816536426544, -0.9196493625640869, -1.4605607986450195, 0.5152708888053894, -0.5111775398254395, 2.256223440170288, -1.0092761516571045, -0.8037801384925842, 0.7934517860412598, 0.42695140838623047, -0.25999346375465393, 0.38545867800712585, -0.8055667877197266, 1.3462382555007935, -1.0701649188995361, -1.0435428619384766, -1.6044304370880127, -0.2813034653663635, -0.4910140633583069, 0.0950063169002533, -1.0655828714370728, 1.057384967803955, -0.32915279269218445, 1.0364007949829102, 0.9951100945472717, -1.6328102350234985, -1.0444997549057007, 1.6042959690093994, -0.6937820315361023, -0.0824674442410469, -0.5570023059844971, -0.7121846079826355, 0.8422194123268127, -0.5167555809020996, -1.47544527053833, -1.4207350015640259, -0.20110443234443665, -0.48110470175743103, 0.570608377456665, 1.8186150789260864, -1.378005027770996, -0.3550202548503876, 0.6611946821212769, -0.10323047637939453, 0.2689395248889923, -1.4022291898727417, 0.8477466702461243, -0.8631986379623413, -1.7234503030776978, 0.49796515703201294, -1.2822281122207642, 0.3018960952758789, 1.1644535064697266, -1.6202276945114136, -1.279128909111023, -2.18436336517334, -0.18192405998706818, -0.6859393119812012, 0.6803034543991089, -0.05919833481311798, -0.21065369248390198, -0.06074977666139603, -0.11706733703613281, -0.6335097551345825, -0.3823391795158386, -1.2899107933044434, -0.2934635877609253, 0.4261932671070099, -0.04561203718185425, 1.860986351966858, 0.5901938080787659, -0.6211461424827576, -0.1630508154630661, 1.5477781295776367, 0.9902951121330261, -1.1156444549560547, 0.023504769429564476, -0.3164448142051697, 0.376166433095932, 1.168893814086914, 0.8245320320129395, -0.3006356954574585, -2.21401047706604, -0.1725437194108963, -0.19867289066314697, 2.0153276920318604, -0.6722347736358643, -0.8165967464447021, -0.9846444725990295, 1.884303092956543, 1.0904722213745117, -0.5621953010559082, 1.4807384014129639, -1.8296680450439453, -1.6528654098510742, -0.8812848329544067, 1.4643809795379639, 0.826087474822998, 0.5088247060775757, 0.37399372458457947, 1.7934579849243164, -1.2935776710510254, -0.5753383040428162, -0.837741494178772, 0.14706607162952423, -0.4494134187698364, 0.01795232482254505, 0.29694032669067383, 0.2616311013698578, -0.7511767745018005, 1.698551893234253, 0.6025837063789368, -1.3398754596710205, -0.08020592480897903, 0.0749349519610405, -1.1413851976394653, 1.2449233531951904, 1.2265596389770508, -1.6005738973617554, 0.8986990451812744, 0.07766559720039368, -0.6234627366065979, -1.3694883584976196, 0.0810806155204773, 2.401658535003662, -1.3157206773757935, 0.9040912389755249, 1.982892632484436, 1.0364220142364502, -0.07195083796977997, -0.7745141983032227, -0.5773931741714478, -0.7822304368019104, 0.1692049354314804, -0.9621256589889526, 0.2386937290430069, -1.2525579929351807, 0.431110680103302, -0.24705828726291656, 0.8645594120025635, 1.6598596572875977, 0.7388864159584045, -0.42219287157058716, 0.2981206774711609, 1.0612421035766602, -0.6069532632827759, -0.9468895792961121, 0.0987481102347374, 0.3415098488330841, -1.0277786254882812, 1.2879709005355835, 0.6465528607368469, 0.8434004783630371, 0.44534289836883545, -0.07842978090047836, -1.270812749862671, -0.9773874282836914, 0.880492091178894, 1.2092775106430054, 0.5637136101722717, 0.23260650038719177, -1.024988055229187, 1.0474753379821777, 0.1544634997844696, 0.7211114764213562, 0.41038548946380615, -0.28620320558547974, -0.17849335074424744, 1.497851014137268, -0.11696228384971619, -0.12532299757003784, 2.232659101486206, 1.1197437047958374, -0.11008047312498093, 0.08436068892478943, -1.411635398864746, -0.15227311849594116, -1.7127643823623657, -1.2230068445205688, 1.2619329690933228, 0.19396445155143738, -0.6383255124092102, -0.7869550585746765, 0.11325429379940033, 0.2875737249851227, 0.8966545462608337, -0.46767187118530273, 0.9979086518287659, 0.5213689804077148, -2.055523633956909, 0.15007132291793823, -2.107848882675171, 0.4828951060771942, -0.05339125543832779, -1.314415454864502, 0.8964723944664001, -0.12563273310661316, -1.1333566904067993, -0.2862561047077179, -1.399216651916504, 0.18576888740062714, 0.3677038550376892, 0.36429816484451294, 0.409223735332489, 1.2936921119689941, -1.219753384590149, 0.6800122857093811, 0.7275161147117615, -0.20410478115081787, -0.8588964343070984, 0.7661295533180237, 0.1050536185503006, 0.7095353007316589, -0.4109838306903839, -0.07932575792074203, -1.096623182296753, -0.3909270167350769, -0.04812946543097496, -0.8717871308326721, 1.4349669218063354, 0.8752933740615845, -0.601008951663971, -1.3548550605773926, 0.449199914932251, -0.18095093965530396, -1.1729810237884521, 0.7605041265487671, 0.9744169116020203, -0.5443803071975708, -0.24678821861743927, 1.7519036531448364, -0.778481125831604, -1.2615054845809937, 0.8759463429450989, -1.2744978666305542, -1.6550350189208984, 0.4923797845840454, -0.40767061710357666, -0.5631429553031921, 0.4652406573295593, 0.312517911195755, 0.321575790643692, -0.5509001612663269, 0.8602274656295776, -0.0980740413069725, -0.19147326052188873, -1.6684460639953613, 0.7397332787513733, -1.2516204118728638, -1.9196021556854248, 0.5278197526931763, 0.5139027833938599, 0.24477654695510864, -0.018342627212405205, 0.8879100680351257, -0.8915618062019348, 0.016951633617281914, -1.0343043804168701, 0.5125342011451721, -1.2069450616836548, 0.16434049606323242, -0.10977867245674133, 0.9228494167327881, -0.23784364759922028, -0.5431026220321655, -0.47439321875572205, -0.0821256935596466, -0.8878328204154968, 1.3304946422576904, 1.9365264177322388, -0.5725916028022766, 0.9221962094306946, -1.446571707725525, 0.7755019664764404, 0.2043227255344391, 0.42179620265960693, 0.29547080397605896, -1.977634072303772, -0.8492081761360168, -0.6674174666404724, -0.8991917967796326, 1.9507888555526733, -0.2834710478782654, -0.659856379032135, 0.23333656787872314, 0.2246287614107132, -0.07287133485078812, -0.4318355619907379, -2.5936288833618164, -1.5842294692993164, 1.7553080320358276, -1.903269648551941, 0.437118262052536, -0.6078867316246033, -2.0140628814697266, -1.8209842443466187, -1.326397180557251, -0.5096473693847656, 0.3666989803314209, -0.9781094789505005, -0.15615123510360718, 0.15246455371379852, -1.3022931814193726, 1.519661784172058, 0.5315641760826111, 0.9331196546554565, 1.3920360803604126, -0.5120400190353394, 1.268223524093628, -1.1023485660552979, 1.5680691003799438, -0.07022590190172195, -0.8609603643417358, -1.0722376108169556, 0.032914917916059494, -0.8554036021232605, -0.05333824083209038, -2.631519079208374, 1.0977468490600586, -0.8896595239639282, 0.2425675243139267, 0.3714485764503479, -2.351328134536743, 0.30246835947036743, -0.2069876343011856, -0.10576766729354858, 1.6878005266189575, -0.23840682208538055, 0.8909519910812378, -0.3597210943698883, 0.4463079869747162, 0.6104642152786255, 0.27868422865867615, -1.1459555625915527, -1.2571340799331665, 1.2951328754425049, -1.6881680488586426, 2.4216015338897705, -1.958063006401062, 0.2528451383113861, -1.1921825408935547, 1.609954833984375, 0.7541766166687012, -0.86838698387146, 0.6435174345970154, -0.6113706827163696, -0.3190406560897827, 0.5294070839881897, -0.0728490874171257, 0.23149771988391876, 0.035704899579286575, -0.5440602898597717, 0.3681706190109253, 0.7938085198402405, -0.6208049654960632, -2.9045190811157227, 1.1499440670013428, -0.19446887075901031, -1.474992275238037, -1.3899548053741455, 1.5271730422973633, -2.495915651321411, 0.4026503264904022, -0.14237737655639648, 1.8005037307739258, 1.1951825618743896, -1.2773538827896118, 0.9102392196655273, 1.6487584114074707, -1.2461342811584473, -0.6140551567077637, -0.23977532982826233, -3.2577924728393555, 0.5654096603393555, 2.2639057636260986, -0.23421284556388855, 0.03875555843114853, 0.6148372292518616, -0.6992198824882507, 0.5602282881736755, -0.4126758277416229, -0.40725329518318176, 0.013810551725327969, -1.4654260873794556, -0.5205944776535034, -2.381659984588623, 0.5885496139526367, 0.3887236714363098, 0.3425534963607788, 0.2872159779071808, -0.30177924036979675, -0.8511202335357666, 1.179451584815979, -0.5347597599029541, -0.882735550403595, -2.151496410369873, -0.43819162249565125, 0.23902466893196106, -2.1634602546691895, -0.4898213744163513, -0.37804117798805237, 0.23963747918605804, -1.4361546039581299, -1.1863734722137451, 0.1370532512664795, -3.2140865325927734, 0.34431198239326477, -0.261972039937973, 1.5655323266983032, -1.3138388395309448, -0.19973716139793396, 1.6044065952301025, 1.501768946647644, -0.27215325832366943, 1.6900275945663452, 0.483254611492157, -0.6175766587257385, -1.9430452585220337, -0.43990767002105713, 0.9491097331047058, -0.33761319518089294, 0.7047663331031799, -1.3057657480239868, 0.23990891873836517, -1.222416877746582, 0.49600154161453247, -0.7221762537956238, -1.8809453248977661, -0.3306870758533478, 2.2274680137634277, 0.5490298271179199, -1.404980182647705, 0.8646036982536316, -0.9865148663520813, -1.738433599472046, 1.4073656797409058, 0.8452327251434326, -0.614653468132019, -1.775426983833313, 0.0071374885737895966, -1.326188325881958, 0.5672569274902344, -0.7973169684410095, 1.5852571725845337, 0.46895039081573486, -0.2878669500350952, 0.268090158700943, 1.1449729204177856, 0.538253128528595, -0.08603990077972412, -0.21771025657653809, -0.4729612469673157, -0.3439207971096039, -0.6528236269950867, -0.4923878610134125, 0.16090133786201477, 1.590865135192871, -0.6627113223075867, -0.35889557003974915, 0.19644348323345184, 0.18329627811908722, -1.1712688207626343, -0.505334734916687, 0.9282624125480652, 3.410334825515747, -1.3417246341705322, -1.1831748485565186, -0.578136682510376, -0.7697766423225403, -1.1919629573822021, -1.4832384586334229, 0.8315797448158264, -0.7265782356262207, -0.021348314359784126, -0.979109525680542, 1.8222235441207886, 1.0488044023513794, 1.4606200456619263, -1.0813264846801758, -0.17961996793746948, -0.24408529698848724, -0.7716631293296814, 1.8088690042495728, 1.8658596277236938, 0.27130162715911865, -0.46263739466667175, 2.067460298538208, -1.9893698692321777, -1.0037122964859009, 0.077884241938591, 0.2027413249015808, 0.6797296404838562, -1.2091615200042725, 0.9942781329154968, -1.206157922744751, 1.6256611347198486, -0.7911834120750427, -0.38315433263778687, 0.6401949524879456, -1.9519463777542114, -1.121368646621704, -0.4842497408390045, 1.1329954862594604, 0.12563614547252655], [1.128480315208435, 0.4637601971626282, -0.7352632880210876, -0.5635015964508057, 0.895756721496582, 0.008598428219556808, 0.23230351507663727, -0.13883720338344574, -0.7834270000457764, 0.5385295152664185, 0.1929478794336319, -1.282827377319336, -0.010771676898002625, 0.6454352736473083, -0.5201517939567566, -0.4909916818141937, -0.5666585564613342, 0.46821072697639465, -0.08523524552583694, -0.7267413139343262, 1.4241586923599243, -0.02175171859562397, 1.2137194871902466, -0.5662270188331604, 1.5564724206924438, -0.9598976373672485, 1.6720398664474487, 1.2786633968353271, -0.009120018221437931, 1.4925719499588013, -0.2324938178062439, -0.709552526473999, -0.747664749622345, -0.6611065864562988, -0.09592043608427048, -0.573506772518158, 0.36908891797065735, -2.804901361465454, -1.6190667152404785, 0.10349604487419128, -0.7293625473976135, -1.355997920036316, 2.8816447257995605, 0.5750992894172668, 0.05382562056183815, -0.16371425986289978, -0.2211591601371765, -0.2065209597349167, 0.38295549154281616, -0.20369361340999603, -1.3880709409713745, -1.4252411127090454, -0.6563010811805725, -0.2171480804681778, 0.4454157054424286, -0.20531554520130157, -1.1777470111846924, 0.0238905418664217, 0.13847649097442627, -0.06747487187385559, -0.4594070017337799, 1.267855167388916, -1.4875116348266602, 1.8236143589019775, -0.5473374724388123, -0.13361623883247375, -1.13838529586792, -1.0479040145874023, 0.33252519369125366, -1.0627225637435913, -0.6755325794219971, 2.5689618587493896, -0.3405330777168274, -1.5866665840148926, 0.176349937915802, 1.6547400951385498, 0.44178327918052673, 0.9159117341041565, -2.92704701423645, 1.2193520069122314, 0.48531487584114075, -0.715999186038971, 0.2886333465576172, -2.3356428146362305, 1.132365345954895, -1.3704837560653687, -1.1550806760787964, -1.8282008171081543, 0.15307117998600006, 0.6631335616111755, -1.059982419013977, -0.509227454662323, 0.42592573165893555, -0.19573549926280975, -1.0370296239852905, 0.6425479054450989, -0.37770625948905945, -1.9337944984436035, 0.6496989727020264, 0.43038952350616455, -0.7100208401679993, 0.6775137186050415, -1.6593141555786133, -1.232852578163147, 1.1300164461135864, -0.06907786428928375, -0.7736878991127014, 0.6153641939163208, -1.9118177890777588, 0.9210465550422668, 0.9000917673110962, -0.23328933119773865, 1.3650633096694946, -0.8778921961784363, 0.04674433916807175, -0.8870895504951477, -0.9031020402908325, 0.4555141031742096, 0.7405900359153748, -1.5832010507583618, 0.0012244185199961066, 0.9673079252243042, -1.4960200786590576, 1.6611242294311523, -1.260387897491455, -1.6890027523040771, 2.314228057861328, -0.3703361749649048, 0.43565651774406433, -0.4077744781970978, -2.511070728302002, -0.3329371213912964, 0.5542553663253784, -0.21826903522014618, -0.835128664970398, -0.42109212279319763, -0.554356038570404, 0.8270044922828674, 0.13620027899742126, -0.9462398290634155, -0.1553085297346115, -1.9815994501113892, -0.2650800347328186, -1.1079250574111938, 0.7579582929611206, 0.2716015577316284, 0.7204453349113464, 1.2557170391082764, -1.8030481338500977, -0.865170419216156, -1.914638876914978, -1.4132486581802368, 0.13812072575092316, 1.2700473070144653, 1.2746607065200806, -1.4216845035552979, -2.0127437114715576, 1.9999332427978516, 0.36923694610595703, -0.8130057454109192, 0.04464728385210037, 1.2278319597244263, -0.783162534236908, -1.1014968156814575, 0.8928279280662537, -1.615978717803955, -0.877366840839386, -0.35613980889320374, -0.382801353931427, 0.7824361324310303, 0.5532599687576294, 0.5433753728866577, -0.6216108202934265, -0.4950597882270813, 2.349170207977295, 0.13540516793727875, 0.4409940242767334, -1.141883373260498, -0.3509139120578766, -1.1327831745147705, 2.2513813972473145, 0.4143825173377991, 0.05527196452021599, -0.67095547914505, -0.9732650518417358, 1.1638925075531006, 1.0042123794555664, 0.12390837073326111, 0.2734728157520294, -1.2044837474822998, 0.5023536086082458, 0.35501280426979065, 0.1580444723367691, -1.5824997425079346, -0.0001886434038169682, 0.8348876237869263, -1.093830943107605, 0.6203534603118896, 0.5384075045585632, -1.815147876739502, -0.5089021921157837, 0.16383813321590424, -0.8363656997680664, -0.5190912485122681, -0.8940083980560303, -2.04219913482666, -2.205329418182373, 0.8943426609039307, 1.8725011348724365, -0.16768892109394073, -0.5303650498390198, -0.32109183073043823, -1.204498291015625, 1.0622059106826782, -0.6026621460914612, -1.1099839210510254, -0.5636807680130005, -0.895766019821167, -0.48897022008895874, 2.068939685821533, 0.38069960474967957, -1.0884191989898682, 0.46315932273864746, -0.2677917182445526, 0.616268515586853, -0.01191466674208641, -0.1342208832502365, -0.08806002140045166, 1.0904709100723267, -1.8179336786270142, -0.1490488201379776, -1.4020651578903198, 0.21373116970062256, 0.05184805765748024, -0.0821387842297554, -0.8880380988121033, 2.0678486824035645, -1.4709792137145996, -1.3671255111694336, -0.22739635407924652, -0.2834153473377228, -1.9437988996505737, 0.34640851616859436, 0.4151727855205536, -1.9049283266067505, -0.00914759561419487, 0.1933462917804718, 1.9691851139068604, -1.915926218032837, 0.2511984705924988, 0.30821675062179565, -1.9118974208831787, 0.5649385452270508, -0.5534515380859375, 0.236201673746109, 0.19612829387187958, -1.0906908512115479, 0.09341131895780563, 1.7148104906082153, -0.41496390104293823, 1.878159761428833, 1.3285026550292969, -0.08585681766271591, 0.38648930191993713, -0.7079753279685974, -1.6128050088882446, 0.8080193400382996, -0.9553978443145752, -2.14359974861145, 1.3575198650360107, -1.0873737335205078, -0.7312234044075012, -0.817579984664917, 0.2832869291305542, -0.488124817609787, 0.7217608690261841, 0.4831617772579193, -0.25426509976387024, -0.17074039578437805, 0.31183817982673645, -0.3334973454475403, -0.31472864747047424, 0.02073436975479126, 0.5244666934013367, -0.4559946656227112, -0.8501380085945129, -0.3914659917354584, -0.6553512215614319, 0.6819735169410706, 0.9601539373397827, 1.07286536693573, 0.1559850424528122, -2.055236577987671, 0.45641639828681946, 0.8933594226837158, -0.45163679122924805, -1.1741900444030762, 1.0252958536148071, -0.240235835313797, 0.24968358874320984, 0.7202314734458923, -0.9524297118186951, 0.513974130153656, 0.6089629530906677, -0.14865009486675262, 1.162570595741272, -0.5034875273704529, -0.7322036623954773, -0.5946729183197021, 0.12440680712461472, 0.032121576368808746, -0.7340074181556702, -0.44917887449264526, -0.32167670130729675, 1.0267601013183594, -0.6058324575424194, -0.16374242305755615, 0.3048434555530548, -0.760601818561554, 0.6348294615745544, -0.9500159025192261, -0.26630035042762756, -0.4439294934272766, -0.6984589695930481, -0.9035492539405823, 0.18884558975696564, -0.9728298783302307, -0.13188204169273376, -0.3074761927127838, -0.4264276623725891, 0.17589963972568512, 0.6243638396263123, 0.3808698356151581, -1.091835618019104, 0.6708188056945801, 0.9361668229103088, 0.49806976318359375, -0.5103959441184998, 1.326414942741394, 0.9720419645309448, 1.9426543712615967, -1.4908347129821777, 0.40084072947502136, 0.4400957226753235, -0.9784202575683594, -0.11132995039224625, -1.2954835891723633, 0.04523971304297447, -0.2606405019760132, 0.2538919448852539, 0.73605877161026, 0.5803167223930359, 1.4455400705337524, 0.32286104559898376, -1.5884668827056885, -0.0656176432967186, -0.6617380380630493, 0.358496755361557, -0.4211916923522949, -0.16544413566589355, 0.3113776743412018, -0.3654763102531433, 0.36462801694869995, 0.4801071584224701, 1.3738949298858643, -0.13291029632091522, -2.4084413051605225, -0.3295173943042755, 0.9003703594207764, -0.013252731412649155, 2.166269063949585, 0.18869291245937347, 0.5929133892059326, -0.9799065589904785, -1.6993944644927979, 0.7805912494659424, -0.5487982630729675, 0.5111680626869202, 0.1579160839319229, 0.36518394947052, 1.0569709539413452, -0.9780194759368896, -0.5231667757034302, 0.7095213532447815, 1.399239182472229, -0.6081029772758484, -0.4650000035762787, 0.06974568963050842, -0.49052342772483826, 0.241041362285614, 0.12530480325222015, 0.8957314491271973, 0.9795531630516052, -0.43780961632728577, 0.29334551095962524, -0.16694729030132294, 0.8701696395874023, 1.2133312225341797, -0.42383652925491333, 1.457085371017456, 0.779090940952301, -0.6853958368301392, -0.07969862222671509, -0.4565965235233307, -0.4869920313358307, 0.31992584466934204, 0.5146916508674622, 0.27556413412094116, 0.38509634137153625, 0.8978812098503113, -1.191462755203247, 1.255362629890442, -2.1993954181671143, 2.0602190494537354, -0.8054841160774231, -1.567867636680603, 0.9659493565559387, 1.0840951204299927, 1.69452965259552, 3.046877384185791, 1.198175072669983, -1.1631048917770386, -0.942872166633606, 0.4219798445701599, 0.45157328248023987, -1.5329645872116089, 0.8179133534431458, -0.8463820219039917, 0.33765485882759094, -0.03731394559144974, -0.932266891002655, -0.6625838875770569, -0.4116203188896179, 1.5024381875991821, -0.685336172580719, 0.5921688079833984, -0.4763026833534241, -0.8760077953338623, -1.0323197841644287, -0.17273612320423126, 1.2646816968917847, -0.15623441338539124, -1.2648836374282837, -0.08811123669147491, -0.8938913345336914, -0.6416232585906982, -1.5184496641159058, 0.6294683814048767, 1.5723402500152588, 0.41992685198783875, -1.1155132055282593, 1.6513946056365967, -0.1903962343931198, 0.34362560510635376, -0.587039589881897, -0.4848373830318451, -0.7201901078224182, -0.8029292821884155, 0.6805416941642761, -1.974490761756897, 0.9473345875740051, 1.204445242881775, 1.505737543106079, -0.5709927082061768, -0.3033188283443451, 1.4218844175338745, -0.2318715900182724, -0.8593748807907104, -0.5113607048988342, -0.30872806906700134, 0.4996418356895447, 1.06832754611969, -1.827045202255249, 0.9015786647796631, -0.6795753836631775, 0.664961040019989, 0.3324424624443054, -1.3356645107269287, 1.6906994581222534, -0.7917566299438477, -0.5038298964500427, -0.8753259778022766, 0.23649711906909943, -1.7356020212173462, 0.6098582148551941, -1.8834409713745117, 1.4095453023910522, -0.1359904259443283, 0.6314453482627869, -1.1967496871948242, 1.048303484916687, -0.03187297284603119, -1.0078567266464233, -1.2253576517105103, 0.47466620802879333, -0.2628503143787384, -0.4898422956466675, 1.134312629699707, 0.675015389919281, -0.4329136610031128, 0.6517854332923889, 0.40105941891670227, 0.9567559957504272, -0.21471160650253296, 0.7990959286689758, 0.6673313975334167, -0.4306395947933197, -3.0279319286346436, -2.3357746601104736, 0.3015555441379547, -0.5698798894882202, -0.9520271420478821, -1.363427758216858, 0.09957255423069, 0.19834423065185547, -0.14689329266548157, 0.3415162265300751, -0.5321036577224731, 1.4984632730484009, 0.3211572468280792, 1.0927441120147705, -1.3031872510910034, 2.0709328651428223, -0.6356702446937561, -2.0687854290008545, -0.03786684200167656, -0.9687148928642273, 0.04520590975880623, 0.37347179651260376, 0.8729411363601685, 1.871407389640808, 0.5042883157730103, 0.018125265836715698, -0.6056416034698486, 1.1605974435806274, 1.7133262157440186, -1.5721017122268677, 0.3894188106060028, -0.8017337918281555, 0.06792342662811279, -1.8766313791275024, -1.0086431503295898, 1.9408046007156372, 0.0236695297062397, -2.2907626628875732, 0.3383311629295349, -0.45928284525871277, -0.15714199841022491, 1.0497517585754395, -2.240386724472046, -1.4716463088989258, -0.8698933124542236, -0.7343780398368835, -0.2559097111225128, -1.226248025894165, -0.3620593547821045, -0.5380164980888367, 0.9548707604408264, -1.2026307582855225, 0.589031457901001, 0.464873731136322, 0.32696476578712463, -0.26724159717559814, -0.621967077255249, -0.030926460400223732, 0.3419988751411438, -0.4860536754131317, -0.5575202107429504, 1.4321857690811157, 0.8285474181175232, 0.6509376168251038, -1.72433602809906, 0.073043592274189, 1.1252577304840088, -0.6567420959472656, 0.4083199203014374, 0.46218469738960266, -0.548564612865448, 1.6018363237380981, -0.6847692131996155, 0.9623697996139526, -0.8166241645812988, 0.7140846252441406, 0.10039614886045456, -0.08895313739776611, -1.3693299293518066, 0.28774988651275635, -2.43620228767395, -0.1653212308883667, 2.441349744796753, -0.30505117774009705, -0.7426526546478271, -0.6769710779190063, 0.3006536066532135, -0.617379903793335, -0.029223324730992317, 0.3055914640426636, -1.5660680532455444, 0.05762004479765892, -1.7824398279190063, -0.2783984839916229, 0.11206596344709396, -0.5972611904144287, 1.1742066144943237, -1.519092082977295, -0.4498254656791687, 1.1506105661392212, -1.5708597898483276, -0.35316359996795654, -2.1875789165496826, 0.7517919540405273, -0.9675871133804321, 2.3533594608306885, -1.3551832437515259, -1.1738637685775757, -1.5331761837005615, 0.7807355523109436, 0.7943294048309326, -1.314180612564087, 1.0244144201278687, 0.15705378353595734, 1.4946132898330688, 0.334808886051178, 1.7292214632034302, 0.003103661583736539, -0.768579363822937, -0.4336182177066803, -0.08557786792516708, 0.8205309510231018, -0.4595436751842499, -0.5612676739692688, -1.2140440940856934, 0.4266892969608307, -0.2575228810310364, -0.4071216285228729, 0.6397239565849304, 0.768105685710907, 0.8695730566978455, -0.5328190326690674, 0.2733829915523529, -0.9213842153549194, 0.08928527683019638, -0.0014266656944528222, 1.9678348302841187, 0.5012571215629578, 0.48016878962516785, 0.42313331365585327, -0.9151557087898254, 0.17597508430480957, 0.7201517820358276, 1.7882722616195679, -0.6002262830734253, 0.41955283284187317, 0.8908388018608093, 0.13363879919052124, -1.3194127082824707, -0.634204626083374, 0.6411178112030029, -0.642391562461853, -0.311629980802536, 0.31384721398353577, 0.19818635284900665, -1.4952222108840942, -1.578276515007019, 0.8524535298347473, 1.1911176443099976, -1.5283507108688354, -0.3039149343967438, 2.3485991954803467, -0.5165907740592957, 0.2722632586956024, -1.0552418231964111, 0.25078892707824707, -0.4506150782108307, -2.1679604053497314, 0.29163286089897156, 0.2921960651874542, 0.3382229804992676, 1.312646746635437, -0.7367321252822876, 0.1307445466518402, -0.339741975069046, -0.24143779277801514, -2.23960542678833, 0.871414065361023, -0.020323486998677254, 1.177435278892517, 2.7802627086639404, -0.6342787742614746, -0.7205589413642883, 0.38208985328674316, 0.04140950366854668, 1.742175579071045, -1.318631887435913, 0.13187400996685028, 2.443142890930176, 0.3931221663951874, 0.33300983905792236, -1.9282946586608887, -0.7949917912483215, 1.8522244691848755, 0.5111505389213562, 0.302054762840271, 0.24758748710155487, -0.8797959089279175, 0.012745077721774578, 0.30580443143844604, 1.6555206775665283, 0.5986554026603699, 1.2866605520248413, -1.5317578315734863, 0.22593186795711517, -1.198920726776123, 0.013256868347525597, 0.540518045425415, -1.5925747156143188, -0.26072177290916443, 0.15641692280769348, 0.645832359790802, 0.09320574998855591, -2.559251070022583, -3.373070478439331, -0.6840529441833496, -0.8770642280578613, 0.45297837257385254, -0.022634334862232208, -2.290048599243164, -1.6770421266555786, -0.49210092425346375, -0.38476285338401794, 0.31683146953582764, -0.17207804322242737, 1.2167409658432007, -0.13127939403057098, -0.03667335584759712, 0.8501056432723999, -1.0891577005386353, -1.8738787174224854, -0.3156091272830963, -0.48957008123397827, -2.0787951946258545, 1.026594877243042, 0.2382134199142456, 0.09960301965475082, -1.4631907939910889, -0.16692985594272614, 0.6111026406288147, 0.8602817058563232, 0.8070053458213806, 0.6016613245010376, 0.12647655606269836, 0.4710055887699127, 1.586600422859192, 1.134574294090271, 0.37523597478866577, 1.3775193691253662, 0.9815842509269714, 0.9352843761444092, -0.3356901705265045, 1.4976904392242432, 0.17796608805656433, 0.5534834265708923, -0.36373791098594666, -0.9440074563026428, -0.6958712339401245, 1.1161366701126099, 0.7146077752113342, 0.674517810344696, -0.8402215838432312, 0.31587669253349304, -1.1587646007537842, 0.5844537019729614, -0.12830914556980133, 0.2759958505630493, -0.08818195760250092, 0.39836421608924866, 0.05863044410943985, -1.6300143003463745, -0.34124886989593506, 2.5100908279418945, 0.3165699243545532, -0.8286647200584412, 1.0016056299209595, -1.335265040397644, -1.0383661985397339, 2.146653652191162, -0.08373458683490753, 0.06464084982872009, -0.2390550971031189, 1.6689974069595337, -0.5186394453048706, -0.35854047536849976, 0.43119046092033386, 0.025578441098332405, 1.0804082155227661, -1.4030513763427734, 1.5482696294784546, -1.1258481740951538, 0.2977263927459717, 1.1354132890701294, -1.1734024286270142, 0.03555276244878769, -1.1586945056915283, -0.9670743346214294, -0.25651371479034424, -0.7835084199905396, 0.25497743487358093, -0.5127720832824707, 0.3429037928581238, -0.053849246352910995, 0.9338729977607727, 1.5353481769561768, -0.06330598890781403, 0.41021615266799927, 1.5322706699371338, 0.7999237179756165, 0.6666339039802551, -0.1615513563156128, -0.568673849105835, -0.8815932869911194, 0.2998705804347992, -0.4591638147830963, 1.013453722000122, 0.25790464878082275, 0.4649970829486847, 0.5014039278030396, 0.3199201226234436, -0.5502524375915527, 1.3609131574630737, 3.089629888534546, 1.5270253419876099, 0.05665513873100281, 2.5690436363220215, 0.14908209443092346, 0.05555272102355957, -0.37987253069877625, 1.4862968921661377, 0.4558887481689453, 0.3388465642929077, 0.3863014280796051, 1.15212881565094, -0.22511467337608337, -0.2081434577703476, 0.27307188510894775, -2.618156671524048, 0.5974230170249939, 0.4663499891757965, -1.4807904958724976, -0.3923405110836029, 1.5952571630477905, -0.13016748428344727, 0.141449436545372, 1.4193291664123535, -0.4638775587081909, -0.4213271737098694, -0.18133333325386047, -0.6069428324699402, 1.8689289093017578, -0.27583253383636475, 1.692326545715332, 0.976783275604248, -0.8214759826660156, -0.6438702940940857, 1.1352027654647827, -0.223293736577034, 0.7702500224113464, -2.9194893836975098, -0.9037744998931885, 1.0158705711364746, -0.26199066638946533, -1.4953900575637817, 0.7388834357261658, 0.021699784323573112, -0.9126718640327454, -0.9082072973251343, -0.49794310331344604, -0.2307472825050354, -0.2231062948703766, -1.450562596321106, -0.35494473576545715, -0.7088279724121094, -0.07081933319568634, -0.6592109799385071, -0.12884043157100677, 0.7450422048568726, 1.5615694522857666, -2.3513875007629395, -0.9045337438583374, -0.6350499987602234, -0.009220674633979797, 1.1498737335205078, 0.10615439713001251, 2.339508056640625, 0.6840519309043884, 0.9599543213844299, -0.9735754132270813, -0.0026755358558148146, 1.2385882139205933, -1.3728986978530884, 0.2455577552318573, -0.1379706710577011, 1.1728039979934692, 1.302302360534668, -0.24399541318416595, -0.4507678151130676, 0.7068280577659607, -0.7626670002937317, -0.0384223572909832, -0.15485697984695435, 1.4023969173431396, 0.8116739392280579, -0.40314823389053345, 0.9622572660446167, -0.5850103497505188, -2.0889995098114014, 0.5383427143096924, -1.1719216108322144, 0.6813115477561951, -2.0289411544799805, 1.488675594329834, -0.6378008723258972, -0.8737656474113464, 0.7917296290397644, 0.9388484954833984, 0.9324761033058167, -1.169731616973877, -0.09493406116962433, -1.5607713460922241, -1.2269905805587769, -1.5213812589645386, 1.6509878635406494, -0.2123425006866455, -1.175443172454834, -0.20677199959754944, -0.20793814957141876, 1.4698200225830078, -1.3863588571548462, 1.5752021074295044, 0.8098637461662292, -0.7153233885765076, 0.22910168766975403, -1.654891848564148, -1.6933014392852783, 0.43836700916290283, -0.24600329995155334, -0.9698717594146729, -0.5099958777427673, -0.09008833020925522, -0.42076992988586426, 0.4662553369998932, 1.2533247470855713, -0.198944553732872, -0.6199145317077637, 0.8756827712059021, -0.20831331610679626, 1.889580488204956, 0.2472347766160965, 0.9788544178009033, 1.760743260383606, 0.9151366353034973, -1.2522531747817993, 0.2291809767484665, -0.8769371509552002, 1.3997282981872559, -0.1747463047504425, 0.1001574695110321, -0.8182340264320374, 1.2410861253738403, -2.1086950302124023, 2.342592477798462, -0.6064718961715698, -0.5690963268280029, 2.2328693866729736, 1.403281569480896, -0.19222603738307953, 0.2183283418416977, -0.167737677693367, 0.4244800806045532, 0.8223995566368103, 0.4327991008758545, 0.3331415355205536, 0.8630487322807312, 1.9533238410949707, -1.1205377578735352, -1.0505292415618896, 0.9708073139190674, 0.3802015781402588, -0.09145721793174744, -0.17175179719924927, 0.4749504327774048, -0.22255109250545502, 0.1736101508140564, -0.3519749939441681, -0.4593292474746704, -0.7733208537101746, 1.306151032447815, -1.1191809177398682, 1.337937355041504, 1.3375767469406128, -0.49004927277565, 0.7320463061332703, 1.4677543640136719, 0.9188053011894226, 0.3213498592376709, 1.1359283924102783, 0.745082676410675, 0.5637347102165222, -1.9158238172531128, -0.38727307319641113, -0.6293228268623352, -0.8772509694099426, -0.9464948177337646, -0.8330762386322021, 0.12053430080413818, 0.9507172703742981, -0.6804718375205994, 0.014639976434409618, -0.18321074545383453, 0.37553834915161133, -1.6174521446228027, 0.2586687505245209, -0.5104196667671204, 0.45751604437828064, 0.6079617738723755], [0.7265889048576355, 0.7006067633628845, 0.2832373082637787, -0.5447453856468201, -0.9342228770256042, -0.578373372554779, 0.5898358821868896, -0.7421836256980896, 1.2291287183761597, -0.5675756931304932, -0.9618493318557739, -1.6842334270477295, 1.3236323595046997, -0.5227575898170471, 1.438907504081726, -1.7410584688186646, -1.3106571435928345, 0.873687744140625, -0.37700459361076355, 0.6456828117370605, -0.018788620829582214, 2.0569045543670654, -0.3998992145061493, 0.7244116067886353, 0.31247377395629883, -0.8690995573997498, 0.47998884320259094, -1.5223029851913452, 1.0479029417037964, -0.11611123383045197, -0.6890937685966492, 1.0280050039291382, 1.934192180633545, -0.5450299978256226, 0.8045816421508789, -0.10019274055957794, -0.35846972465515137, 0.22562424838542938, -0.909839391708374, 1.7284904718399048, -1.1903811693191528, 0.8738751411437988, -0.6374210119247437, -1.4812756776809692, -0.297742635011673, -0.12191950529813766, 0.6850423216819763, 1.2150200605392456, -1.8192920684814453, 0.47482043504714966, -2.1747076511383057, -1.5371876955032349, 0.5705570578575134, 0.4011845290660858, -0.7429442405700684, -0.37700510025024414, 0.34563276171684265, 0.07103519141674042, -1.192597508430481, 0.9733209609985352, -0.6304943561553955, 0.304048091173172, 0.10022854804992676, 1.1358120441436768, 1.1915130615234375, -0.47922369837760925, -0.5929314494132996, -0.08077553659677505, 2.2225310802459717, -0.0169939324259758, 0.012425155378878117, 1.611688494682312, 1.210362195968628, -0.5982311964035034, 0.5007781386375427, -0.5815205574035645, 0.47095462679862976, 0.12578007578849792, -0.6747710108757019, -0.09883671253919601, 0.713945209980011, -1.138031005859375, -0.8277787566184998, -0.5677165985107422, 1.012856125831604, -0.8377707600593567, 0.6134043335914612, -2.356973171234131, -0.3786012828350067, -0.33955907821655273, 1.909403920173645, -0.7494663000106812, -0.20845207571983337, 0.5056031346321106, 0.7801257371902466, 1.5046334266662598, 0.16837024688720703, 0.8866834044456482, 0.8086125254631042, 1.5519760847091675, -0.4833871126174927, 0.5774169564247131, 0.9807328581809998, -0.5728061199188232, -1.157616138458252, 0.11340773105621338, 0.43584293127059937, -1.5191824436187744, -0.41914963722229004, -0.3552848994731903, -0.6867338418960571, -1.2429072856903076, -0.7943101525306702, 0.38752540946006775, -0.09014994651079178, 1.038143277168274, -1.5262466669082642, 0.04432692751288414, -1.5956029891967773, -0.21284203231334686, 1.537541389465332, 0.692947506904602, 1.094313621520996, -1.4715148210525513, -0.028612371534109116, 0.7464607954025269, -1.5400962829589844, 0.28689587116241455, -0.7755597233772278, 1.6835434436798096, -0.686596155166626, -1.4289430379867554, 0.7374998927116394, -0.9271721839904785, 0.36817651987075806, 1.0252389907836914, 1.7996975183486938, -0.8259502649307251, 0.5029968023300171, 1.012188196182251, 0.10721836239099503, 1.1196544170379639, 1.0797276496887207, 0.42800474166870117, -0.04272649437189102, 0.7489016056060791, 0.8585744500160217, -2.019151449203491, -0.3703482151031494, 0.5787679553031921, -0.03306940570473671, -0.6872925758361816, 0.3498592972755432, 0.6114503145217896, -1.6309211254119873, 0.48565545678138733, 0.18821614980697632, -0.5510858297348022, 1.1844162940979004, 0.17007267475128174, -0.5765874981880188, 0.9812482595443726, 1.1906239986419678, 0.04170886427164078, -1.65557062625885, -0.3728136420249939, 0.12857860326766968, 0.9967713356018066, -0.2609531581401825, 2.5389819145202637, -1.8750007152557373, -1.5284276008605957, -1.6881401538848877, -0.6210928559303284, 0.2417753040790558, 1.042533278465271, 0.3637223541736603, 0.22609862685203552, 0.32954686880111694, 0.302062064409256, 0.3485165536403656, 0.1878654807806015, -0.5738821029663086, -0.35308510065078735, 1.370599389076233, 0.3577244281768799, 1.5264540910720825, 0.33601465821266174, -0.8930880427360535, -1.1431798934936523, 0.8660057187080383, -0.27202072739601135, -0.6706382632255554, 0.04342228174209595, -1.9251439571380615, 2.0325167179107666, 0.6977419853210449, 0.0984983742237091, 0.4629128873348236, -0.32428550720214844, 0.25773847103118896, 0.8886492848396301, -1.0538489818572998, -0.6900725960731506, 1.1063222885131836, 1.1884019374847412, 0.8759142756462097, -1.6413697004318237, -1.2287801504135132, 0.23100866377353668, -0.25081416964530945, -1.9425556659698486, 0.23404783010482788, -1.0474579334259033, -0.5274500250816345, 0.17173074185848236, -0.8426532745361328, 0.20915339887142181, 0.030937757343053818, -0.571212112903595, -2.1989591121673584, -0.3711986243724823, -0.015394255518913269, 0.4536329209804535, 0.5896109342575073, 0.01604657992720604, -1.4598530530929565, -0.9071389436721802, 0.4836367666721344, -0.061102863401174545, -0.928058922290802, 0.5294337272644043, 0.9091953039169312, -0.3116688132286072, 0.8321067094802856, 1.8601951599121094, -0.16689851880073547, -0.17703722417354584, 0.9409205317497253, -0.6654634475708008, 1.468430519104004, 1.1033787727355957, 0.9081915616989136, -1.8562121391296387, -0.5947155952453613, -0.46195507049560547, -0.8642677664756775, -0.2502017319202423, 1.3546249866485596, 0.07114679366350174, 0.1510021984577179, 0.6147267818450928, 1.4007633924484253, -1.2474772930145264, 0.24586661159992218, -0.017629535868763924, 0.38315561413764954, 1.3670068979263306, 1.1354199647903442, 0.5650540590286255, -0.7331655025482178, -0.9498327374458313, -1.2841700315475464, -0.8013970255851746, 1.5683144330978394, 0.5105878710746765, -2.181759834289551, 0.5117037296295166, -1.5674028396606445, -0.09027600288391113, -0.6430845856666565, 1.9879041910171509, 0.4791813790798187, -0.3805866539478302, -0.446631520986557, 1.5460013151168823, 0.3687921166419983, -0.8019965291023254, 0.27505338191986084, -0.9881157279014587, 0.35839325189590454, 0.9622035026550293, 2.0317225456237793, 0.7524340152740479, 1.084876537322998, 1.7504454851150513, -2.053293228149414, -1.6873539686203003, 0.27724745869636536, 1.0211045742034912, -0.7290762066841125, -0.3178645372390747, -0.7435427308082581, -1.4362297058105469, 0.9235992431640625, -0.06699623912572861, 0.1663414090871811, -0.16040343046188354, -1.2409350872039795, -0.3505861163139343, -0.5685858726501465, 0.7599729299545288, -0.1368015557527542, -0.6746793389320374, -0.7535665035247803, 0.4425632059574127, -0.8655127882957458, -0.820214033126831, 0.7411684989929199, -1.3261293172836304, -0.36835619807243347, -1.3797452449798584, -0.7880196571350098, -0.6531015038490295, 0.5228840708732605, 0.8208098411560059, 0.44444039463996887, -1.0647801160812378, -0.8446944952011108, -1.4270588159561157, 0.5182117819786072, 1.2057994604110718, -1.0622060298919678, 0.8583917617797852, -0.7427542209625244, 0.4225924611091614, 0.768267810344696, -1.3308035135269165, -1.9220434427261353, -1.0199391841888428, 0.7720372676849365, 0.9401550889015198, -1.4609496593475342, -0.039013542234897614, -0.8639670014381409, 1.2668907642364502, 0.07454772293567657, 0.6295742988586426, 0.6618337631225586, -0.5237032771110535, -0.3602803945541382, -0.3629211187362671, -0.03979188948869705, -1.2155828475952148, -0.3654210865497589, -2.9262003898620605, 0.732856810092926, -2.136808395385742, -0.4429425895214081, 0.049164097756147385, 0.8372659683227539, -0.5833207964897156, -0.5276247262954712, -0.970315158367157, -0.4598131477832794, -0.5451526045799255, 0.26808616518974304, -1.6697196960449219, -1.6925139427185059, -0.251541405916214, 1.2408603429794312, -0.5239005088806152, 0.21844223141670227, -1.678350567817688, 0.18964657187461853, 0.4634569585323334, 0.03465607762336731, 0.6469482779502869, -1.2130924463272095, 0.628639280796051, 0.7954665422439575, 0.12221734970808029, 1.1689715385437012, 0.9358595013618469, -0.7933922410011292, -0.5210980772972107, -0.8816197514533997, -0.7950549721717834, -1.2981065511703491, 0.846140444278717, -0.36273571848869324, 1.0344104766845703, 1.1106009483337402, -0.40167057514190674, 0.11632594466209412, -1.4597781896591187, 0.976862370967865, -0.893612265586853, -0.5461089611053467, -0.3379719853401184, 0.19765214622020721, -0.7508937120437622, -1.587597131729126, 1.5271047353744507, -0.4313987195491791, -0.04298543557524681, 1.5306512117385864, -1.4091010093688965, -0.19640333950519562, -0.18951956927776337, 1.1446782350540161, -0.06733428686857224, -2.1975386142730713, -0.1771232932806015, -1.0847089290618896, 0.8614517450332642, -0.3609270453453064, -0.684330940246582, 1.450177788734436, -0.11980709433555603, -1.072784185409546, -0.6112805008888245, 0.13337931036949158, 1.1628299951553345, -0.07243245840072632, -0.9372997879981995, -0.48557984828948975, 2.196742057800293, 0.053681619465351105, -0.9833685755729675, 0.38016706705093384, 0.3677816092967987, -0.3070223331451416, -0.024930128827691078, 1.6780974864959717, 0.6885647177696228, 0.3576088547706604, -0.9097726941108704, 0.14453816413879395, 1.443530559539795, -1.5967798233032227, 0.08308131247758865, -0.35727640986442566, -0.905634343624115, 2.0101685523986816, 0.7526746988296509, 0.25519824028015137, 0.39204272627830505, -0.1020083874464035, -0.5005533695220947, -0.9076937437057495, 0.4643816351890564, -1.4183008670806885, -0.9318143129348755, -0.7453570365905762, 1.8081063032150269, 1.489713430404663, -0.3508043587207794, -1.707135796546936, -0.15499910712242126, 0.4685746729373932, -2.1019983291625977, 0.7976903319358826, -0.10428285598754883, 0.2865540385246277, -0.3950711786746979, -0.6688613891601562, -2.8328857421875, -0.8473100662231445, -0.41707509756088257, -0.8382367491722107, 0.6539384126663208, -0.6438414454460144, -0.35485711693763733, 0.20152470469474792, -0.22908881306648254, 1.0486037731170654, 0.7793427109718323, -1.902411699295044, -1.288239598274231, -0.8628163933753967, -0.3410417139530182, 1.3122515678405762, 0.6890519857406616, -0.7380455732345581, 2.3736355304718018, -0.5061026215553284, 0.24400146305561066, -1.5960195064544678, -0.17577795684337616, 1.5780420303344727, -0.4171924591064453, -0.8416533470153809, 1.9839916229248047, 1.1148372888565063, -1.0858774185180664, -1.0202096700668335, -2.561626672744751, 0.455240398645401, -0.08499753475189209, 0.757229208946228, 0.3125340938568115, -0.9631686210632324, 1.0644696950912476, -0.7310053706169128, -0.6977832317352295, -0.8794170022010803, 0.601969301700592, 0.35945484042167664, -1.5927979946136475, -0.34087955951690674, 1.4001775979995728, 0.7908044457435608, 1.100637435913086, 2.3126280307769775, -0.9452611207962036, 1.4407967329025269, -0.18128164112567902, -0.6193574070930481, -0.3174685835838318, -0.6119791865348816, 1.1525710821151733, 0.05718378350138664, -1.228245496749878, -1.5474671125411987, -0.46389198303222656, 0.5597541332244873, 0.09193581342697144, 0.14640362560749054, -2.5145866870880127, -0.5268716812133789, 1.4156240224838257, 0.22875024378299713, -1.2855888605117798, -0.9069934487342834, 0.24246959388256073, 0.04164471477270126, 0.3561953604221344, -1.5432815551757812, 0.76300048828125, 1.5344855785369873, 0.1073509082198143, -1.2333012819290161, -0.10716097801923752, -0.07614060491323471, 0.580450713634491, 0.7719097137451172, 0.8855169415473938, -0.23416022956371307, 1.8624212741851807, 1.523816466331482, -0.13011011481285095, -0.152336984872818, -0.5437097549438477, -0.9105132818222046, 0.3577970266342163, 0.5629361867904663, -0.9717385172843933, 0.5198068618774414, 1.600873589515686, 1.392999529838562, -0.8500345349311829, -0.2385461926460266, 1.0365352630615234, -1.286893367767334, -0.4964462220668793, -1.0868204832077026, -0.42352086305618286, 0.26219484210014343, -0.41787856817245483, -0.715329110622406, -1.1398935317993164, 1.1624045372009277, 1.8425524234771729, 0.45734575390815735, -0.9933762550354004, -1.971187949180603, -2.121387004852295, 0.5041164755821228, -2.136678695678711, 0.42018043994903564, -0.9063618779182434, -0.5032249689102173, 0.340495765209198, -0.11047309637069702, 0.24296370148658752, -0.588949978351593, 1.0588326454162598, 0.244513601064682, 0.2818405330181122, 1.2718627452850342, 0.06944580376148224, -0.5605542659759521, -0.6754788756370544, 1.324522852897644, 1.7987513542175293, 0.4655199646949768, 1.2103842496871948, -0.33903229236602783, -1.6621685028076172, 1.401415228843689, -0.5650639533996582, 0.30642229318618774, 1.6530777215957642, 0.6317551136016846, 0.8793180584907532, -1.6623454093933105, 0.5464658141136169, 0.25595542788505554, -0.4136330783367157, 0.6121070981025696, 0.28055524826049805, -0.18389376997947693, 1.3242369890213013, 0.7217817902565002, -0.5050280094146729, 0.22496381402015686, -0.6287320852279663, 0.032604895532131195, -0.18630196154117584, -0.47342827916145325, -0.9415648579597473, -0.22755740582942963, 0.03773569315671921, -0.9051162004470825, 0.7760501503944397, -0.7880887389183044, -0.6193885207176208, -0.4050888419151306, 0.5454002022743225, 0.8228371143341064, -1.295354962348938, 0.01585153490304947, -0.34472066164016724, -0.6989859938621521, -1.8929834365844727, 1.3397055864334106, -1.718090295791626, 0.0483730174601078, 1.5462677478790283, -1.9409193992614746, -0.5833582878112793, -0.38811445236206055, -0.20994757115840912, -0.8106039762496948, 1.9097051620483398, 0.37018725275993347, -1.1543192863464355, 0.25737079977989197, 0.44203463196754456, 0.6125401258468628, 0.45793697237968445, 0.09144852310419083, 0.21556007862091064, -0.7823963165283203, -1.718299388885498, 0.18610617518424988, 0.07329010218381882, -1.2983604669570923, 0.45821958780288696, -0.5710334777832031, -0.691567063331604, -0.3371742367744446, 1.626688838005066, 0.3992665410041809, -0.5373091697692871, 1.812370777130127, -0.5886381268501282, 1.0132973194122314, 0.25948262214660645, -0.8768870234489441, 0.3138638138771057, -0.43851715326309204, -0.8976494669914246, -0.035679180175065994, -0.7363566756248474, -0.46227002143859863, 0.5524793863296509, -1.0270805358886719, -1.3789913654327393, 0.7541999816894531, 0.018385281786322594, 1.0356696844100952, -0.8257214426994324, -0.16499875485897064, 0.308554083108902, 0.6669594645500183, -0.9940541386604309, 1.0004111528396606, -0.7589260339736938, 0.026673581451177597, -1.5981346368789673, 0.42738664150238037, -1.3451915979385376, 0.44040820002555847, -0.29283133149147034, 0.29236915707588196, -1.619036316871643, 1.0073429346084595, -0.09325648844242096, -0.02800595760345459, -0.3089916408061981, -1.1512333154678345, -0.153387650847435, -0.11027025431394577, 0.9628095030784607, -0.748568058013916, 0.19508211314678192, 0.9646024703979492, 0.9332798719406128, 1.051613211631775, 0.7182387113571167, 0.2508680522441864, -1.270958423614502, 0.034837812185287476, -0.5706200003623962, 1.3872917890548706, -0.06514835357666016, 2.082706928253174, 1.1294312477111816, 0.1591310203075409, -0.03225862979888916, -0.7858688831329346, 2.2958338260650635, 0.5091161131858826, -0.22541308403015137, -0.685714066028595, -0.8317639231681824, -0.6897928714752197, -1.5517394542694092, -0.6324630975723267, 0.8983675241470337, 1.4941216707229614, -1.256238579750061, -0.1619471162557602, -1.0890412330627441, -0.7158916592597961, 0.32722848653793335, -0.5001383423805237, 1.2325875759124756, -1.2384724617004395, 0.5126535892486572, -0.7441977858543396, -0.2663813829421997, 0.3013285994529724, -0.028317268937826157, 0.07654084265232086, 1.2034703493118286, 0.6872565746307373, -0.19957466423511505, 0.5879855155944824, 1.6031558513641357, -0.6760565638542175, -0.07592258602380753, 0.5047941207885742, 0.05554762855172157, -0.5626903176307678, -0.5190816521644592, 0.17314007878303528, -0.4465251564979553, 0.3328154683113098, -1.4672337770462036, 0.8836398124694824, -0.01110551692545414, -0.18755711615085602, -0.7877148389816284, -0.8893031477928162, 1.4240583181381226, 0.89131098985672, -0.6509791016578674, -1.0526765584945679, 0.45658859610557556, -0.4671821594238281, -1.3833385705947876, 1.669411063194275, 0.6378454566001892, 0.2819633483886719, 1.6534669399261475, 0.7636663913726807, -0.45300033688545227, 0.34554293751716614, -0.11538922041654587, 0.08089666813611984, 0.3456389904022217, 0.46597597002983093, -1.858119010925293, 0.5897310972213745, 0.47819989919662476, 0.2459021508693695, -1.4226205348968506, -1.9960143566131592, 0.10133074969053268, 1.1045584678649902, -1.7153457403182983, -0.6515375971794128, 1.317915916442871, 0.888593852519989, 1.6836507320404053, -0.5035907030105591, -0.10694119334220886, 1.288291573524475, -0.43219414353370667, 0.991615891456604, 1.342319130897522, 1.6713440418243408, -0.21864764392375946, 0.6020436882972717, 1.2622623443603516, 0.7003375291824341, -1.0062191486358643, 0.4027305245399475, -0.054092392325401306, -1.4936723709106445, 0.7794117331504822, 1.5813548564910889, -1.7251567840576172, 0.03519225865602493, 0.17683513462543488, -1.1331684589385986, -0.9224730730056763, -0.0453207828104496, 0.904744029045105, 0.18202488124370575, 1.3342857360839844, -0.728035569190979, -0.36510851979255676, -1.354816198348999, -0.8294702768325806, -0.7594330310821533, -2.281703472137451, -0.5389944911003113, -0.4940244257450104, 2.543989896774292, -0.12951794266700745, 0.7838627099990845, -1.1406052112579346, -1.8094860315322876, -0.9431270360946655, 0.6298766136169434, -0.19447514414787292, 0.6701933741569519, -0.12435868382453918, -1.9805195331573486, -1.2083592414855957, 0.3401057720184326, -0.6889317631721497, -0.460918664932251, 0.13998007774353027, 0.6306700110435486, -0.26797008514404297, 0.5329180955886841, -0.9822263717651367, 0.27046066522598267, 0.19391809403896332, 0.7677093148231506, -0.05798789858818054, -1.3204724788665771, -1.5399067401885986, -0.4329318404197693, -0.046757977455854416, 0.38253557682037354, -0.4842251241207123, 0.27778953313827515, -0.5701920390129089, -0.06244558095932007, -0.5630372762680054, 0.13353048264980316, 0.5649320483207703, 1.2072031497955322, -1.2939647436141968, 1.6169649362564087, -0.2869289815425873, 0.6693507432937622, 0.46192502975463867, 0.9272125959396362, -1.1497265100479126, -1.5170396566390991, -0.07444792985916138, -1.4366130828857422, 0.5741440057754517, 1.184181809425354, -1.25499427318573, 0.48739221692085266, 0.03907943516969681, 0.08485279977321625, -0.9400618672370911, -1.1418894529342651, -0.975834310054779, -0.3010239601135254, 0.5203761458396912, -1.112496018409729, 0.0858011469244957, -0.33923062682151794, -0.8195696473121643, 0.9937158823013306, -0.41899096965789795, -1.827711582183838, -0.5582308769226074, -0.7299201488494873, -0.010488731786608696, 1.6491734981536865, -0.07270536571741104, -0.3590274155139923, -0.1625906080007553, 0.3272072970867157, 1.510074496269226, 0.3480168581008911, 0.3442302346229553, -0.16515035927295685, 0.38512226939201355, -0.6013050675392151, 1.8885152339935303, -0.5256184935569763, 0.21379400789737701, 0.6417169570922852, -0.9876556992530823, 0.046995632350444794, 2.687957525253296, 0.5087969303131104, 0.40505850315093994, -2.7834227085113525, -0.7842172980308533, -1.0478851795196533, 0.34623876214027405, -0.31577473878860474, 1.1626193523406982, -1.5863559246063232, -0.20526103675365448, -1.383608102798462, -0.457743376493454, -0.23264841735363007, 0.8404584527015686, 0.10933930426836014, 0.035055357962846756, -0.20103712379932404, 1.396824836730957, 0.8797780275344849, -0.4683407247066498, 0.5537739992141724, 0.13768945634365082, -0.6937676668167114, -1.059153437614441, -1.1986287832260132, 0.6359137296676636, 0.5569158792495728, 0.12532612681388855, -1.0404446125030518, -0.7836366295814514, 0.9841605424880981, 0.3794831931591034, -0.5613745450973511, 0.6860150098800659, 1.0003224611282349, -0.8831421732902527, 0.9750156998634338, -0.21114838123321533, 1.6932392120361328, 0.36286935210227966, -1.8499042987823486, 0.4161491394042969, -0.03645152598619461, -0.2925527095794678, -0.08801385015249252, 1.2420021295547485, 0.5692749619483948, -0.8236026763916016, -1.30596923828125, 0.6950133442878723, 1.8615888357162476, 0.015085829421877861, -0.014196052215993404, 0.6186206340789795, -0.24901899695396423, -0.21640191972255707, 0.9264201521873474, 0.3460707366466522, -1.591554880142212, 1.0215235948562622, -0.7835858464241028, 0.36165425181388855, -2.943953514099121, -2.4728095531463623, -0.2884523570537567, 0.119893878698349, 0.8835026025772095, 0.2572806477546692, 0.1934199333190918, 0.870413601398468, -1.3971011638641357, 1.7609835863113403, -0.0887107402086258, 2.0746140480041504, -0.25578761100769043, -0.569185197353363, 0.781657874584198, -0.9417008757591248, 0.7555367946624756, -0.4851721227169037, 2.5948572158813477, -0.23281478881835938, -0.7277536988258362, 0.4295269846916199, 1.3544496297836304, -0.637064516544342, -1.5295062065124512, -0.07038533687591553, -0.01201858650892973, -1.8304883241653442, 1.1998668909072876, 0.9489437341690063, 0.8214516639709473, 0.989192008972168, -0.70988929271698, 1.383070945739746, -0.8272601366043091, -0.701024055480957, -0.6539602279663086, -0.1888028085231781, -2.424846649169922, 0.7824545502662659, -0.8212124705314636, 0.27490395307540894, -0.3717842400074005, -1.279578685760498, -0.9175301790237427, 0.5414220690727234, 0.5818615555763245, -0.44121024012565613, 0.4112078547477722, 1.484707236289978, 0.5048038363456726, -0.5695966482162476, -0.18597769737243652, -0.0041159773245453835], [0.232765793800354, -1.9207913875579834, -0.704658031463623, -1.053597331047058, 1.7297677993774414, -0.8836842775344849, 0.7750145196914673, 1.5699158906936646, 0.5328065752983093, 0.8848268389701843, 1.836443305015564, -2.219097852706909, -0.6874990463256836, -1.8366260528564453, -0.3169538676738739, -0.6374714970588684, -0.4166763722896576, 1.0499335527420044, 0.545774519443512, 0.05995878204703331, -1.5731972455978394, -0.7538305521011353, 0.6768898963928223, 0.4803845286369324, -1.2082186937332153, -0.07918665558099747, -0.25673291087150574, 0.6679835319519043, 0.09002840518951416, -2.1482603549957275, -0.08632797002792358, 1.379467248916626, -0.8489794135093689, -1.216195821762085, -1.9015851020812988, -0.38217681646347046, -0.19956561923027039, -0.8491096496582031, 0.3708091378211975, -0.5293875932693481, -1.528004765510559, 0.9432040452957153, -0.5917713642120361, 0.9474822878837585, -0.8815189003944397, 0.04712691158056259, -0.993159294128418, -0.7757993340492249, 0.9888975024223328, -1.382784128189087, -0.48243051767349243, -1.966797947883606, 0.4925304055213928, -1.3881703615188599, -1.115260362625122, -0.22612783312797546, 0.16490881145000458, 1.4519656896591187, 0.5246294736862183, 1.744324803352356, 1.3716543912887573, 0.020841458812355995, -1.4571541547775269, 0.7668280005455017, -0.2650131583213806, -0.054974477738142014, -1.1206196546554565, 0.18599601089954376, -3.195948600769043, -1.0298690795898438, -0.3627011775970459, 1.33646821975708, -2.082169771194458, 0.4909382164478302, -0.3464498817920685, 0.6316342949867249, -0.7130016088485718, 0.8369702100753784, 0.9933496117591858, 1.1889504194259644, 0.6129890084266663, 0.28501421213150024, -0.06610284745693207, 0.3452325463294983, -2.254328489303589, -0.5526319742202759, 0.24013382196426392, 0.8858814239501953, -0.5862755179405212, 0.47960352897644043, -0.4960360825061798, 0.23516908288002014, -0.22323855757713318, -0.3541989028453827, -0.17565858364105225, 0.6105371117591858, -0.7592895030975342, -0.7424258589744568, 0.8715338110923767, 0.7571887969970703, -2.724806308746338, 0.8913878798484802, -1.7396997213363647, -1.3360689878463745, 0.6900892853736877, -0.3881932497024536, -0.4845298230648041, 1.178564429283142, 2.450756072998047, 1.0727670192718506, 0.5571969151496887, 0.0470317117869854, -2.6750240325927734, -2.012855052947998, 0.43976297974586487, 2.3619625568389893, -0.28428810834884644, 0.7761431932449341, -1.1459693908691406, 0.03391915559768677, 0.5860329270362854, -0.4286452829837799, -2.353388786315918, 0.5034717917442322, -0.5995414853096008, -0.8682520389556885, -0.14039233326911926, 4.162094593048096, -0.8628443479537964, 0.32602640986442566, -0.09577399492263794, -1.812657117843628, 1.7848093509674072, -0.9677168726921082, 1.319504737854004, -0.36692893505096436, -0.11543168127536774, 1.1793619394302368, 0.1984838992357254, -0.890849232673645, 0.7145504951477051, 0.01610233075916767, 0.25609543919563293, 0.4984279274940491, -1.1015982627868652, -1.3498737812042236, 0.1540711373090744, 0.5608832836151123, -1.0434402227401733, 0.9651196599006653, -0.0888177901506424, 1.7581374645233154, -1.5096898078918457, -1.2422008514404297, 1.8679940700531006, 0.8640693426132202, -2.6554203033447266, 1.2287209033966064, -1.2631171941757202, 2.257758617401123, -0.07021375745534897, -0.8992421627044678, 0.3603529930114746, 0.02125624753534794, 0.16215947270393372, 0.7374847531318665, -0.7879177331924438, -0.2659240961074829, 0.4264239966869354, 0.7675845623016357, 1.6167181730270386, 0.8396409749984741, 0.34175512194633484, 0.5737860202789307, -0.3097461760044098, 0.31123587489128113, 1.1351081132888794, -0.5943477153778076, -0.5483381748199463, -0.8471588492393494, 0.5108800530433655, 0.043368153274059296, 0.4733343720436096, 1.2784844636917114, -1.6026175022125244, 0.4664338231086731, 0.17854461073875427, -1.0223406553268433, 0.6486406326293945, -0.7200493812561035, -1.5609455108642578, 0.94143146276474, 0.5691172480583191, -0.68534255027771, 0.3561420738697052, 1.0444095134735107, 1.671990990638733, -1.5939017534255981, -1.2874066829681396, -2.251739740371704, 0.16348017752170563, 0.2618747353553772, -0.2902805805206299, -2.5322680473327637, 0.5980280041694641, 0.4996647536754608, 1.4382401704788208, -1.0341600179672241, 0.667579710483551, 0.7231091856956482, 1.0946396589279175, 2.789797782897949, 0.07219655066728592, 0.8402150273323059, 0.10352654755115509, 0.8894446492195129, -1.4554131031036377, 0.10571200400590897, -1.4795795679092407, 1.7168060541152954, 0.7460067868232727, 0.8837339282035828, 0.1461496204137802, 0.9906324744224548, -0.040730737149715424, -1.0688352584838867, 0.2612464427947998, -0.973391056060791, 0.39925721287727356, -1.6796871423721313, 0.13618971407413483, -1.861398696899414, -0.5429587364196777, -0.0047433520667254925, 0.35088035464286804, 1.0559651851654053, 0.9105423092842102, -0.506588339805603, 0.1439283937215805, -0.6568508744239807, -0.9267669916152954, 1.6805930137634277, -0.3301054537296295, -0.2902640700340271, -2.0982308387756348, -1.0029258728027344, 0.3224116265773773, 0.9262107610702515, -0.6484561562538147, -0.7722516655921936, -1.3981332778930664, 2.4576363563537598, -0.23573996126651764, 0.08034317940473557, -0.6752059459686279, 0.615084707736969, -0.6418271660804749, 1.3876088857650757, -1.6429493427276611, -0.21652387082576752, 1.2633495330810547, -0.8875101804733276, -0.39401379227638245, -0.02265540137887001, -0.5052762627601624, 0.2779398262500763, -0.5764588117599487, -0.8255077600479126, -1.0943567752838135, 0.10276620835065842, -0.3355003297328949, -1.8010578155517578, -1.0305793285369873, 1.2555975914001465, 0.05553416162729263, 0.6578124165534973, 0.581311047077179, -0.28697195649147034, -1.1914258003234863, -0.8436325192451477, 1.2796626091003418, 0.39893582463264465, 0.348174512386322, 1.0149294137954712, 1.7754120826721191, 1.746253252029419, -0.22978630661964417, -0.26933735609054565, 2.034856081008911, 0.01835089921951294, 1.8940308094024658, -0.18143874406814575, -0.003220045007765293, -0.6118016839027405, 0.8215823769569397, -0.8244168162345886, -1.1611127853393555, -0.46380361914634705, -0.24279223382472992, -1.1426669359207153, 0.8894709348678589, 0.3033704459667206, -0.38205215334892273, 0.9339905977249146, -0.5654447674751282, 0.14568845927715302, -0.773517370223999, -0.5524899363517761, 2.793654203414917, -0.42554351687431335, 0.6772243976593018, 0.15955005586147308, -1.5469799041748047, 0.09835249930620193, -0.18056176602840424, -1.2650580406188965, -0.30901047587394714, 0.975487232208252, -0.5399843454360962, -1.6053797006607056, 0.9854384064674377, 0.13175800442695618, -0.24447160959243774, -1.8408368825912476, 0.9225532412528992, -0.15708063542842865, -0.17158359289169312, 1.3897780179977417, -0.25192588567733765, 0.4491901695728302, -0.5744034647941589, 2.392252206802368, -0.7473188042640686, -0.9357187151908875, -0.0373731292784214, -1.651653528213501, 0.8829531073570251, 0.22088396549224854, 1.3775615692138672, 0.7480576038360596, -1.8532066345214844, 0.7926395535469055, -3.2700302600860596, -0.625898003578186, 1.483545184135437, -0.4736252725124359, -0.8806228041648865, 1.5153859853744507, -0.5995739102363586, 0.7490559816360474, 0.37658509612083435, -0.29659152030944824, -0.1543959081172943, 1.0776594877243042, -0.411281019449234, 1.0498238801956177, -0.7274447679519653, 3.2020652294158936, -0.1107407659292221, -0.14061443507671356, 0.4801487922668457, 1.4214940071105957, 0.338861346244812, 0.24549442529678345, 0.8195434808731079, 1.4577038288116455, -0.9539740085601807, 0.06220655515789986, 1.8032671213150024, -0.5750830173492432, -0.3310062289237976, 0.42460039258003235, 1.2828457355499268, 0.014841857366263866, -0.42575618624687195, -0.523439347743988, 0.04113345593214035, -0.4718073308467865, 0.57298743724823, -0.4434657096862793, 0.5770372152328491, 0.065202996134758, 0.9436565041542053, -1.2380989789962769, 0.42111915349960327, -1.3129312992095947, 0.724337637424469, 1.1116896867752075, 1.4380285739898682, -0.8008342981338501, 0.6716214418411255, -0.5782767534255981, -0.14402960240840912, -0.1519848108291626, -0.3317667543888092, -0.5026785731315613, 1.2974252700805664, -1.6211731433868408, 0.01004450861364603, 0.17444492876529694, 1.157212734222412, -0.2574070990085602, 1.028030514717102, 1.8464151620864868, -1.0332486629486084, -0.472101628780365, 0.4781574010848999, -1.4038041830062866, -1.3048737049102783, -1.0975677967071533, -0.5947245955467224, -1.5381089448928833, 0.9929099082946777, -1.4228612184524536, -0.9094831347465515, -0.9450049996376038, 0.068272165954113, -0.6982111930847168, 0.9654909372329712, -0.3429635763168335, 0.6744380593299866, -1.1582237482070923, 1.5762851238250732, -1.891101598739624, -0.15381444990634918, 0.32896092534065247, -0.273669570684433, -1.241947054862976, -1.323046326637268, -0.19841136038303375, -1.2769614458084106, -0.2237958461046219, 0.1389000415802002, 0.31069687008857727, -1.1059361696243286, -1.2059321403503418, -0.25869861245155334, 0.8695772886276245, 0.5918817520141602, -1.261417031288147, 0.8732119202613831, 0.6767263412475586, 0.3928760588169098, -1.4955819845199585, -1.3822027444839478, -1.013580560684204, -0.7603357434272766, 0.2878560721874237, 0.7272881865501404, 0.8468893766403198, -2.25925874710083, 1.1961191892623901, 0.734815239906311, 0.5577387809753418, -0.8281694054603577, -1.950514554977417, 0.468506783246994, 0.019815396517515182, -1.1767711639404297, -0.25238731503486633, 0.7445350289344788, -0.6435683965682983, -0.3652406334877014, -0.5643064379692078, 0.8384186625480652, -0.8534815311431885, -0.21505337953567505, -0.4978139102458954, -1.0553083419799805, -0.2492428421974182, -0.21893437206745148, -0.7476575970649719, -0.07341166585683823, -0.9189962148666382, 1.552550196647644, -0.15049739181995392, -0.5640363097190857, -0.20429642498493195, 0.39382168650627136, -1.8182387351989746, 0.9667149186134338, 0.39312300086021423, -0.9390079379081726, 1.4639776945114136, 0.36010316014289856, -0.26201099157333374, -0.5470186471939087, 0.8830733299255371, -0.6208442449569702, 0.17549745738506317, 0.012910328805446625, -0.1373506784439087, -0.7285272479057312, -0.5130002498626709, -0.22567445039749146, -0.7427839040756226, -0.14143714308738708, 1.051915168762207, -0.3927784860134125, 0.18882834911346436, 0.6598098874092102, 0.44112029671669006, -0.44664058089256287, -0.7564147710800171, 1.5620986223220825, 0.025903047993779182, -1.1838499307632446, -1.4237624406814575, 1.3388400077819824, 0.48741158843040466, 0.04476071894168854, 1.1387983560562134, -0.636163055896759, -1.9204293489456177, -0.9627789258956909, 0.9679077863693237, -0.10057950019836426, -0.3007989823818207, -0.8816251754760742, 0.3469572961330414, -0.04500434175133705, 0.6932041049003601, -0.8357555270195007, -0.034336742013692856, 2.417903423309326, -0.29953622817993164, -0.12566372752189636, 0.049999456852674484, -0.49935540556907654, -0.28128480911254883, -0.19146078824996948, -1.0855790376663208, -0.48203057050704956, -0.9094612002372742, 0.9946308732032776, 0.5393386483192444, 0.5973716378211975, 1.3660598993301392, 0.5012254118919373, -1.058427333831787, 0.8181772828102112, -2.019751787185669, -0.6486741900444031, -0.9179178476333618, 0.5229202508926392, 0.8772598505020142, -1.4024745225906372, -0.0007533450843766332, 2.2542033195495605, -0.21471752226352692, 0.5943878889083862, 1.2958122491836548, 0.26223254203796387, 0.7814177870750427, -0.2511875331401825, -0.14916323125362396, -0.1407890021800995, 0.03874671831727028, -0.913082480430603, 0.9073943495750427, -0.4872586727142334, 1.9120678901672363, -0.8684908747673035, 0.21656495332717896, 0.43139687180519104, -1.6971397399902344, 0.012976194731891155, 0.33110934495925903, -0.13904894888401031, -1.1996386051177979, 0.5714020133018494, -0.10645917057991028, 0.3157324492931366, 0.37123918533325195, 0.4229867458343506, -0.4717079997062683, -0.4875408113002777, 0.09068188071250916, -0.03525099158287048, 0.5552204251289368, -0.576082706451416, 1.378811001777649, -0.036083873361349106, 1.266927719116211, 1.5706981420516968, 0.9785105586051941, 0.22945789992809296, 0.05642891675233841, -0.03203756362199783, 0.9314123392105103, 0.6165978312492371, -0.2782885432243347, -2.3599112033843994, -0.9509244561195374, -1.979576826095581, -0.8231222033500671, 0.5322039127349854, -0.20026390254497528, 1.1578656435012817, 0.3098209500312805, -0.18946592509746552, 2.034304618835449, -1.0550341606140137, -1.3048557043075562, 0.9840774536132812, 0.025909701362252235, -0.45498254895210266, 0.4977247416973114, 0.31156525015830994, 1.3649593591690063, -1.741611123085022, -1.0812851190567017, 1.6944737434387207, 1.2615259885787964, -0.6067550778388977, -1.474484920501709, 0.8657904267311096, -1.2094570398330688, -0.5729548931121826, 0.0562533363699913, -0.18180358409881592, 0.3952498137950897, 0.013011487200856209, 0.9251033067703247, -0.21331802010536194, 1.0159298181533813, -0.6711968779563904, 0.675838828086853, -1.2945998907089233, -0.3769751489162445, 0.17050692439079285, -0.3978859782218933, 3.1119186878204346, -1.1545372009277344, -0.8299762606620789, -0.0023449757136404514, 1.1125552654266357, -1.497726321220398, -0.593513011932373, 1.7961540222167969, -1.1359026432037354, -0.18326787650585175, -1.5178052186965942, 0.29910120368003845, -0.036381304264068604, -0.4793243408203125, -1.9600855112075806, 0.4667067527770996, 0.3327220678329468, -0.033167287707328796, 1.6148611307144165, -0.12393861263990402, 0.623386561870575, 1.2266325950622559, 1.1546986103057861, 0.19182558357715607, 0.6278506517410278, -0.46166887879371643, -0.3383980691432953, 0.559022068977356, -0.19882608950138092, 1.1334782838821411, 0.9767779111862183, 0.5831232666969299, -0.8128410577774048, 0.3513388931751251, 0.39742588996887207, -0.03998083248734474, -0.3186470866203308, 0.2634441554546356, 0.3362801969051361, 0.5044946074485779, 1.9545382261276245, 0.6464895606040955, -0.5182576775550842, -1.2548682689666748, 0.4421895444393158, 0.12190237641334534, -0.9305241703987122, -1.4430439472198486, -1.125599980354309, 0.2978178858757019, 0.4611743986606598, 1.120859980583191, 0.748858630657196, 1.0833076238632202, -1.7362953424453735, 0.20569899678230286, -1.3084381818771362, 0.9525455832481384, -1.3702203035354614, 0.1313788890838623, 0.18969053030014038, 1.4382365942001343, -1.3483697175979614, -0.9784942269325256, 0.8313136100769043, 2.0159530639648438, 0.20557118952274323, 0.06801902502775192, -1.1744332313537598, 0.5234861969947815, 0.45882946252822876, 2.1089587211608887, 1.4096198081970215, 1.4196019172668457, 0.03259795904159546, 0.7660860419273376, 1.8117238283157349, -0.4462316632270813, 0.8280120491981506, 0.7353683114051819, -1.6814360618591309, 0.8223626613616943, 0.06610773503780365, -0.027351997792720795, -0.7337945103645325, -0.18284831941127777, -0.35313668847084045, -0.02958400547504425, 0.9851070642471313, -2.006812572479248, -0.13919959962368011, 1.1183534860610962, 1.6045962572097778, -0.18595220148563385, 0.043840810656547546, 1.2710051536560059, -0.13939715921878815, 0.2054339051246643, -0.5226675868034363, -1.3787965774536133, 0.825926661491394, -0.799589216709137, 1.1512387990951538, -2.369485855102539, -0.35716941952705383, -0.1406002789735794, 0.3479326069355011, -0.8913791179656982, -1.4593359231948853, 0.713401198387146, 0.0974159762263298, 0.367181658744812, 0.21134218573570251, -0.14099985361099243, -0.8633435368537903, -0.49289482831954956, 0.8593493103981018, 1.238627552986145, -0.030003370717167854, -0.8561573028564453, -0.6750195026397705, 0.6186380982398987, -0.5151867866516113, -0.2232465147972107, -1.7717418670654297, 0.2505381405353546, 0.6050348877906799, 0.3430637717247009, 1.9155832529067993, -0.23496784269809723, 1.254282832145691, 0.6100759506225586, 1.0319424867630005, 0.7460022568702698, -1.1579439640045166, -0.08316007256507874, -0.6385708451271057, 0.6715742945671082, 1.5073901414871216, -0.5269019603729248, 0.8384807705879211, -0.2285524606704712, -0.38442564010620117, 0.14193512499332428, 0.9468199610710144, -0.10154678672552109, 0.2352973222732544, -0.9918327331542969, 0.5284808874130249, -0.08879166096448898, 1.3524835109710693, 0.26567542552948, 0.441924124956131, -1.5587079524993896, 2.1350483894348145, 1.31569504737854, 0.8067501783370972, -0.49550819396972656, -0.26138395071029663, -2.675344228744507, -0.16506074368953705, 0.24778766930103302, 1.1657960414886475, -1.1648139953613281, 0.4500811696052551, -1.4216068983078003, -0.9374738335609436, 1.6414560079574585, 0.104496531188488, 1.1471353769302368, 0.29787611961364746, 1.506274700164795, 0.8731085658073425, -1.3270262479782104, 0.6172881722450256, 1.541412591934204, -1.2440136671066284, -1.2971570491790771, -0.07860579341650009, 1.3862842321395874, 0.7751814723014832, -0.0670497789978981, 0.15103061497211456, 2.220836639404297, 0.27520644664764404, 0.4047640860080719, 0.3722236752510071, 0.35686108469963074, -0.9126899242401123, 0.31701716780662537, -1.2383393049240112, -0.1603255718946457, 2.9037160873413086, -0.7773857116699219, -2.1611781120300293, 0.043887801468372345, 0.2456127256155014, 0.012295574881136417, -0.7245470285415649, 0.7804038524627686, -1.2889045476913452, 0.03501949459314346, 1.2205923795700073, -1.062759518623352, -1.3927364349365234, 1.8155763149261475, 0.1114756390452385, 0.26021093130111694, 1.0937525033950806, 2.426280975341797, 0.7347617745399475, -1.2933863401412964, -1.2200405597686768, -1.1245836019515991, 1.4528371095657349, -1.2299906015396118, 0.7873228788375854, -0.1872287094593048, 0.9841316342353821, 0.6050960421562195, 0.6528764367103577, -0.2692495882511139, -1.1914247274398804, 0.3682348132133484, -1.8254374265670776, -1.0435954332351685, -0.5560745596885681, -0.33592405915260315, -0.6818093657493591, -0.43534615635871887, -0.17922094464302063, 0.3031396269798279, -1.6105133295059204, 2.1624608039855957, -1.5345569849014282, 0.4674496054649353, 0.15252184867858887, 0.10543286055326462, -1.0376248359680176, 0.49006760120391846, 0.4651266932487488, 0.5385646820068359, 0.0698787271976471, -0.34757766127586365, -0.807716429233551, 0.9999303817749023, 0.8037136793136597, 0.8174917101860046, -0.29337048530578613, -0.3066210448741913, 0.718450665473938, -0.2464670091867447, 0.6043766736984253, -1.919414758682251, 2.7610409259796143, -0.6493313312530518, -1.2124598026275635, -0.417784184217453, 1.1385242938995361, 1.7299848794937134, 0.5350325107574463, -0.3644806444644928, -0.5294970273971558, 0.27648836374282837, 0.7719446420669556, -1.5476638078689575, -0.00040852383244782686, 0.8000497817993164, 1.5339726209640503, -0.42352619767189026, -1.3477615118026733, 0.8309541344642639, 0.6426771283149719, 1.7443469762802124, -0.7138534784317017, -0.5316035747528076, -0.3345750570297241, -0.0987970232963562, 1.928543210029602, -1.404158592224121, 0.47455787658691406, 0.03846896439790726, -0.3297300934791565, -1.2806956768035889, 0.4790359139442444, 0.8119615316390991, 0.08220235258340836, 0.6792892813682556, 0.5371398329734802, 0.06819384545087814, -1.414262056350708, 1.003493309020996, 0.249745711684227, -0.1772530972957611, -0.061780981719493866, -1.1145844459533691, 0.48335912823677063, -0.46043166518211365, 0.9922277927398682, 0.3577083945274353, -1.4859005212783813, -1.1466453075408936, 1.5891709327697754, -0.3751034438610077, 1.016120433807373, -0.22878582775592804, -0.44729623198509216, 1.5455167293548584, -0.44135305285453796, 0.46363353729248047, 0.8514336943626404, -1.3281623125076294, 0.16050291061401367, -0.37958747148513794, 1.138529658317566, 0.8717459440231323, -1.8680124282836914, -0.5133976340293884, -0.7680609822273254, -2.020167827606201, -0.4803781509399414, -1.1332591772079468, -0.7178947925567627, -0.2494087815284729, -0.15561847388744354, 0.6383392810821533, -0.8326608538627625, -1.4927400350570679, 0.3463481068611145, -0.5715970993041992, 0.5563803911209106, -0.8913659453392029, -0.3138202726840973, -1.256752371788025, -0.0747823417186737, 0.322410523891449, -0.5809676647186279, 0.43138256669044495, 0.791166365146637, 1.4516366720199585, 0.06612823158502579, -0.7454223036766052, -0.23020365834236145, 0.12726831436157227, 0.552750825881958, -0.55589759349823, -0.7625765800476074, 0.034998390823602676, -0.003634090069681406, 0.10243310779333115, 0.014979051426053047, 0.20426715910434723, -0.011445295065641403, -0.3979306221008301, 0.20230460166931152, 0.7763275504112244, -1.9813294410705566, -0.44544297456741333, 0.20491531491279602, -0.1278371661901474, -1.5244084596633911, -0.31357496976852417, -0.6682630181312561, -1.1922062635421753, 0.2555865943431854, -0.8014317750930786, 0.6973971724510193, -1.362008810043335, 0.44680020213127136, 2.6096596717834473, 1.40366530418396, 1.1152946949005127, -0.48254603147506714, -0.08475597947835922, 0.16903991997241974, -0.8850656747817993, 1.0390346050262451, 1.2093679904937744, -0.6282846927642822, 0.3067851662635803, 2.288079023361206, 0.46338024735450745, 0.6621796488761902, -0.6816301941871643, -1.231141209602356, -2.322159767150879, -0.2283947616815567, -0.1747085601091385, -0.9885636568069458, -1.96561861038208, -1.2960174083709717, -1.0726144313812256], [-0.6979443430900574, 0.7484780550003052, 1.075507640838623, 1.4446825981140137, -1.4182151556015015, -1.2425551414489746, -0.4429130256175995, 0.7615095973014832, 1.2745674848556519, -0.304818332195282, 0.7366868257522583, 0.7248632311820984, 2.0266642570495605, 0.20243176817893982, 2.484732151031494, -1.111002802848816, -0.05871319770812988, 2.132007360458374, 0.28586965799331665, 0.0427473783493042, -0.07804529368877411, 0.15614642202854156, 2.080986499786377, 1.1475454568862915, -0.5430362820625305, 1.4807003736495972, 0.27382180094718933, -0.1966060996055603, -0.32644712924957275, 1.1150870323181152, 0.6905201077461243, 0.7319778800010681, 1.1597681045532227, 0.43198421597480774, 3.3424196243286133, 0.7552782297134399, 0.7296417951583862, -0.8246195316314697, -0.05574648082256317, -1.3516929149627686, -1.5779744386672974, -0.23683491349220276, 0.500178337097168, -0.028739191591739655, 0.4332234561443329, 0.8443389534950256, -0.9708236455917358, 0.38770613074302673, -1.1157262325286865, -1.9871221780776978, 0.7129178643226624, 0.5536589026451111, -1.2578362226486206, -0.5186822414398193, 1.1257458925247192, 0.9394051432609558, 0.30584198236465454, -0.7938093543052673, 1.4805593490600586, 1.0277446508407593, 2.556051731109619, -0.6946122050285339, -0.4700687527656555, 0.916235625743866, 0.17710423469543457, -0.5598562955856323, 1.1013001203536987, 0.9090412259101868, 1.1736000776290894, 0.25780487060546875, 1.4997947216033936, -0.22988742589950562, -0.864679217338562, 0.35570865869522095, 0.8346429467201233, 0.07243522256612778, 1.3691742420196533, -1.8746404647827148, 2.3196868896484375, 2.01181960105896, -0.9368897676467896, -0.22059081494808197, 1.0345174074172974, 1.07265043258667, -0.03371211886405945, 0.6816239953041077, 0.0424790158867836, -1.521762728691101, -1.3219411373138428, -0.2705944776535034, 0.023956691846251488, -1.505721926689148, -1.5881385803222656, -0.6427401900291443, -1.5540623664855957, -0.7840587496757507, -0.5362686514854431, -1.3663294315338135, -1.3246550559997559, 0.6570644378662109, 0.2054695039987564, -0.14118121564388275, 1.817766547203064, -0.290816068649292, 0.0997796580195427, -1.7723408937454224, -0.8570341467857361, -0.003943640273064375, 0.861617922782898, -0.25278666615486145, 1.2335278987884521, -1.668715000152588, 0.9681872129440308, -1.345029592514038, -0.1555899679660797, 0.19816701114177704, -0.888502299785614, -0.19291453063488007, -2.2404696941375732, -1.2794498205184937, 1.7658041715621948, -0.09873086959123611, 0.015457692556083202, 0.18510645627975464, 1.3323636054992676, -2.0543317794799805, 1.296419382095337, -0.6303656101226807, 0.2870754599571228, -0.471141517162323, 0.08787353336811066, 0.08702393621206284, 2.1387264728546143, 0.1441621035337448, 0.7069497108459473, 1.554985761642456, 0.3592611849308014, 0.4271966218948364, 0.46602994203567505, -1.206845998764038, 1.5024256706237793, -0.3508729338645935, 1.4242291450500488, 0.38852280378341675, -0.45950448513031006, -0.18554486334323883, 1.272249460220337, 1.249610185623169, 0.2994261682033539, 0.15627743303775787, 0.5337443351745605, 1.0008951425552368, -0.7850717306137085, -2.113090753555298, -0.08963492512702942, 0.9803740382194519, -0.5753286480903625, 0.4210434854030609, 1.532405138015747, -0.22450734674930573, -0.3917725086212158, -1.3148729801177979, 1.3513425588607788, -0.6087135076522827, 1.8046200275421143, 0.13791467249393463, 0.16556483507156372, -0.6342015266418457, -0.32938551902770996, 0.8882063627243042, -0.14263327419757843, -0.25273171067237854, -0.9170415997505188, 0.1343247890472412, 0.3049998879432678, -1.7949349880218506, 1.2249358892440796, 0.029995586723089218, 0.006210395600646734, 0.390103280544281, 0.5815070271492004, 1.8360910415649414, 0.5439766049385071, 1.4098174571990967, 0.6479570865631104, 1.4250776767730713, 0.36337241530418396, 0.7503618597984314, 1.0030790567398071, -0.5677371621131897, -0.5886905193328857, 0.06871218234300613, -0.009707883931696415, -0.07049505412578583, -1.484871506690979, 0.7617026567459106, -1.4623923301696777, 0.8710756897926331, 0.6258983612060547, -0.3555881083011627, 1.9769333600997925, -1.0510631799697876, 1.9307365417480469, 1.1978678703308105, -0.14522439241409302, -1.8914896249771118, -0.10503683984279633, 2.1575589179992676, -1.9850084781646729, 1.3009198904037476, -1.3301918506622314, -2.7768850326538086, 0.4959593117237091, -2.332463264465332, -0.3942933678627014, -0.9221926927566528, -0.189341202378273, 0.16821931302547455, 1.166366457939148, 0.08531571924686432, -0.5286127924919128, -0.231620192527771, 0.8109686374664307, 0.38125279545783997, 0.5461027026176453, -0.21333494782447815, 0.5490185618400574, -0.8061366677284241, -0.021101681515574455, -0.9577374458312988, 1.9314186573028564, -1.451642632484436, 0.34901687502861023, -1.5994104146957397, -0.747417151927948, -1.5336532592773438, -1.0715522766113281, -0.3668392598628998, -0.7725616693496704, -0.4636945128440857, 1.742201328277588, -0.024138348177075386, -0.8227367997169495, 0.8019280433654785, -1.9181817770004272, 0.605006754398346, -0.6973021030426025, -1.69731605052948, -0.8375080823898315, 0.21076644957065582, -0.17436495423316956, 0.2113649845123291, 0.8200281858444214, 1.2308064699172974, 0.7929816842079163, -0.5642572045326233, -1.0770782232284546, -0.18907372653484344, -0.6998699307441711, -0.3883911967277527, 1.0720769166946411, -2.5441665649414062, 0.16256184875965118, 0.07772655785083771, -0.6398124098777771, -0.34699946641921997, 1.2607554197311401, 0.17165929079055786, 0.0006291365716606379, 0.29949110746383667, -0.6419832110404968, 0.03762679547071457, 0.7410458922386169, -0.20629453659057617, -0.0983729362487793, 0.6852973699569702, 0.5825474262237549, -2.0026190280914307, -0.7742975354194641, -1.831427812576294, -0.7282959818840027, 0.35712912678718567, 1.2673267126083374, -0.47129857540130615, -2.3362221717834473, 0.18485215306282043, 1.4555879831314087, -1.422118902206421, 2.356090784072876, -2.5060477256774902, 1.6329752206802368, -1.5320689678192139, -0.25043097138404846, 2.4350271224975586, -0.5264317393302917, 2.201683521270752, -0.6206148266792297, 1.9775912761688232, 0.04596336558461189, 0.8626160621643066, 1.0521245002746582, 1.2435195446014404, 1.4702165126800537, 0.6129133701324463, 0.1727534830570221, 0.7056756615638733, 1.5991522073745728, -2.0410706996917725, 0.303226500749588, 0.9452820420265198, -0.7950164675712585, -0.00785801187157631, -0.0864400565624237, 0.539341151714325, 1.267386555671692, -0.3084503412246704, 2.00974440574646, 0.4112583100795746, -0.9691504240036011, 0.12755870819091797, -0.671373188495636, -1.5446140766143799, -0.08016699552536011, -0.8710005879402161, 1.6153950691223145, 0.008692340925335884, -0.8366777300834656, 0.42666417360305786, -1.5379592180252075, 0.7811727523803711, 0.20117105543613434, 0.7677393555641174, -0.8299263715744019, -0.8548716306686401, 1.0881688594818115, 1.6810933351516724, -0.43115848302841187, -1.265315294265747, 2.6361236572265625, -0.8644794821739197, 0.743037760257721, -0.7271751761436462, 0.6214959025382996, -0.6720013618469238, -3.025538682937622, 0.7534784078598022, 1.6950912475585938, 0.2930811941623688, -1.1385939121246338, 0.28461459279060364, 0.046840790659189224, 1.0038601160049438, 0.577103316783905, 0.7005236148834229, -0.28730034828186035, -0.34949687123298645, -0.6491844058036804, -1.1145802736282349, -0.903416097164154, 1.4567526578903198, -0.0171800684183836, -0.12013821303844452, -0.8991214632987976, 1.5396513938903809, -0.873222291469574, -0.5743820667266846, 0.27809521555900574, -2.4351415634155273, 0.8107816576957703, 0.7778549194335938, -0.8608121871948242, 0.5304150581359863, 0.6333024501800537, 1.4786531925201416, -1.1906737089157104, -0.879548192024231, 0.36206263303756714, 0.70473712682724, -0.35043656826019287, -1.876449465751648, -0.43460991978645325, -1.7263938188552856, -0.5216234922409058, 0.6751483082771301, 0.06191262602806091, 1.317618727684021, 0.05914970859885216, -2.0044891834259033, -0.08462841063737869, 0.9440048336982727, -2.050713300704956, 0.2896425724029541, -0.2688103914260864, -0.054502807557582855, -0.10870927572250366, -0.6809019446372986, -1.556086778640747, 0.07805638015270233, -1.3079103231430054, 0.30161017179489136, -0.7066690325737, 1.8444793224334717, 1.1519789695739746, -0.4863104820251465, 0.31076890230178833, 0.6262041926383972, -1.6218011379241943, -0.16797147691249847, -0.01624978706240654, 0.23871202766895294, 0.966300904750824, -0.9340450763702393, 0.03171277418732643, 0.654260516166687, -0.12156637012958527, 0.4782584309577942, -0.7297908663749695, -2.2870163917541504, -0.5832827091217041, 0.13854916393756866, -0.08762940019369125, -0.25619497895240784, -0.1924804300069809, -0.6669114232063293, 2.047086238861084, 0.5826605558395386, -0.3069210350513458, 0.36802002787590027, 1.5823843479156494, 1.0851364135742188, -2.640026330947876, 0.3437010943889618, -0.5552767515182495, -0.8442109227180481, 0.23895084857940674, 1.7383313179016113, 1.5020862817764282, 0.42443206906318665, -0.7867819666862488, -0.3384510278701782, 0.5758342742919922, 0.02575434185564518, -0.623930811882019, -0.42611998319625854, 0.002221307484433055, -0.5219533443450928, -0.020241299644112587, -0.4666041135787964, 0.713245689868927, 0.21020828187465668, -0.37437704205513, 0.5775436162948608, -0.8675041794776917, 0.491842657327652, -0.9447624087333679, 0.41117531061172485, -0.03359401971101761, -2.1349234580993652, 1.434447169303894, -1.924620509147644, 0.05050230398774147, -0.05423123389482498, -0.2868664264678955, 1.9434714317321777, 0.18322335183620453, 1.2861480712890625, -0.1679619550704956, -1.053878903388977, 1.1217727661132812, 0.22537535429000854, -0.18644696474075317, -1.0169330835342407, -0.20971845090389252, 2.431751251220703, -0.08373326808214188, -0.25368890166282654, -0.41181787848472595, -0.48989957571029663, -0.47013595700263977, -1.1296623945236206, 0.5349860787391663, 0.24496231973171234, -0.39021798968315125, 0.8093448281288147, -0.530836284160614, -0.4553389251232147, 0.9241130352020264, 0.43312960863113403, 0.33648666739463806, 0.2566071152687073, 1.7110124826431274, -0.5578117966651917, -0.5021382570266724, -0.8575466871261597, -1.2222599983215332, -1.2726784944534302, -0.06942083686590195, -0.27294236421585083, 0.48693832755088806, -0.008944080211222172, 0.670559287071228, 0.6164851784706116, 1.115551233291626, -0.519711434841156, 0.24556560814380646, 0.03910703584551811, -0.5531139373779297, -2.0898404121398926, 1.3340901136398315, -2.06392765045166, -0.1980406939983368, 1.677822232246399, 0.3751015067100525, -0.6185489296913147, -0.11549441516399384, 1.2062779664993286, 1.6582159996032715, 0.8490109443664551, 1.3440529108047485, -0.6579939126968384, -0.10756037384271622, -0.5527122616767883, -0.3136867880821228, 0.035326037555933, 0.5944771766662598, -0.2999207079410553, 0.48143139481544495, 0.45202523469924927, -1.6917059421539307, 0.7827777862548828, 1.3310534954071045, 1.2772005796432495, -0.3538902699947357, -0.11033819615840912, 0.8927373290061951, -0.6112648248672485, -0.5837197303771973, -0.23817478120326996, -0.3907299339771271, 0.25899738073349, 0.24202635884284973, -0.3622281849384308, 0.6621072888374329, -1.4146313667297363, -0.1140977218747139, 0.10071124881505966, -0.4830490052700043, 1.0820270776748657, 0.003842244390398264, -1.0383810997009277, 0.21015195548534393, -0.39757516980171204, 1.8789912462234497, -0.6829501390457153, -0.37855851650238037, -0.17441484332084656, -1.0604125261306763, -0.47397229075431824, -0.932087242603302, -0.9568648934364319, -1.2905149459838867, 0.7362109422683716, -1.8897234201431274, 0.12416486442089081, -2.0730669498443604, -0.8001718521118164, -0.8728382587432861, -0.30396193265914917, -2.1943540573120117, 0.5260045528411865, 0.2532087564468384, -0.2019731104373932, -0.7395265102386475, -1.377503752708435, -1.7109984159469604, 1.2482670545578003, 0.1131807491183281, -0.7198193073272705, 0.9554072022438049, -0.36462536454200745, -0.20809930562973022, -0.5675724148750305, -1.612084984779358, -1.6078834533691406, 0.5851118564605713, -2.131521224975586, 0.7807250022888184, -0.990648627281189, 1.038196325302124, -1.7676162719726562, -1.4290404319763184, 0.010625460185110569, 1.7825729846954346, -0.677547037601471, -0.17980194091796875, 2.1594016551971436, 1.5345544815063477, -0.46882978081703186, 3.4661989212036133, -0.33277469873428345, 0.5147570371627808, -0.2351028025150299, -1.9266011714935303, -1.3315987586975098, -1.063460350036621, 0.11779636889696121, -0.9056342840194702, -0.4559800922870636, 1.7558587789535522, 0.815894365310669, -1.726523756980896, -0.4582536816596985, -0.10926288366317749, 0.7697858214378357, -0.11860044300556183, -0.7085803747177124, -1.5740221738815308, 0.4168838560581207, 1.1334658861160278, 0.25508442521095276, 0.029933249577879906, -1.0358452796936035, 0.6026034355163574, -1.756633996963501, 0.5062412619590759, 1.0566396713256836, -0.29032930731773376, -1.161737322807312, 0.9655682444572449, -0.24967685341835022, -0.16556932032108307, -1.2013815641403198, 0.6405515074729919, -0.2970063388347626, -0.5285505652427673, 0.36230891942977905, 0.4393111765384674, -0.45212146639823914, 0.5636996030807495, -0.6978934407234192, -1.0280495882034302, 1.1023924350738525, -0.5309181809425354, -0.6739704012870789, 1.1138219833374023, -0.5247337818145752, 1.0501189231872559, -0.2480987161397934, 1.167583703994751, 0.5533654689788818, 0.7830933332443237, -0.2514333426952362, 0.9357928037643433, -0.15318650007247925, -0.1624467968940735, 1.809765100479126, -1.12135910987854, 1.1715352535247803, -0.14437492191791534, 0.4496624171733856, -1.023421049118042, -0.8184664845466614, 0.527462899684906, -0.5244874358177185, -0.5223178267478943, -0.024027245119214058, 0.8135358095169067, -0.9399902820587158, 1.2921011447906494, 0.4797145426273346, -0.411930650472641, 1.1263353824615479, -0.9118016362190247, -0.18606019020080566, -0.7205032706260681, -1.714303970336914, -0.38165438175201416, 0.8448737263679504, -0.19614417850971222, -1.529194951057434, -1.273695945739746, -1.2729452848434448, 0.6167597770690918, -0.8126873970031738, -0.12797516584396362, 1.6857532262802124, -1.0032645463943481, -0.8539263010025024, -1.5314781665802002, -0.2972286343574524, 1.0626935958862305, 1.4662288427352905, 1.0068862438201904, -0.711427628993988, -0.0803237184882164, -0.6323751211166382, -0.03140272945165634, 0.08945725113153458, 0.8715851902961731, 0.8596014976501465, -0.8054388165473938, 0.6089600324630737, -0.3550480902194977, 0.9008933305740356, 0.47024863958358765, -0.5558286905288696, 0.1647537499666214, 0.4291497766971588, -0.075391486287117, -0.43431293964385986, -0.32597965002059937, -0.159963458776474, -1.0764609575271606, -0.8111376166343689, 1.5586329698562622, -0.42328765988349915, -0.16667139530181885, -0.17640987038612366, 0.22362719476222992, -0.47917792201042175, -1.2577298879623413, 0.08020871877670288, -2.3947224617004395, -0.35898730158805847, -1.8414732217788696, 1.9600059986114502, 0.7250455617904663, 0.5735567808151245, 0.6164941191673279, 1.8203800916671753, -0.7251206636428833, 1.0693607330322266, 0.7485328316688538, -0.28858762979507446, 2.126976490020752, 0.22053804993629456, 1.1918201446533203, 0.9141220450401306, -0.21345777809619904, 0.13599447906017303, 1.2531899213790894, -1.632865309715271, 0.572597861289978, -1.3207323551177979, -1.1601749658584595, -0.0013909906847402453, 0.9927674531936646, -0.5749700665473938, -0.45669227838516235, 1.320310354232788, -1.1133487224578857, -1.9486474990844727, -0.9777457118034363, -0.20209386944770813, 1.9596980810165405, -0.3337143659591675, 0.5389734506607056, 0.025530733168125153, 0.6662754416465759, -0.2085120677947998, -1.3347663879394531, -0.6295018196105957, 0.8498557806015015, 0.35988178849220276, 0.09765169769525528, -0.08333975076675415, 0.27802062034606934, 1.4405465126037598, -0.6975042819976807, 0.7918544411659241, -0.6783995628356934, -0.7788289785385132, 0.029568104073405266, -0.619351863861084, -0.0675826296210289, 0.906089186668396, 0.47247716784477234, -0.9709662795066833, 0.6188061833381653, -1.5340782403945923, 0.7642725110054016, -0.9779435396194458, 1.3824784755706787, -0.6875879168510437, -0.42511892318725586, -0.012680349871516228, -2.272975444793701, -0.4110372066497803, 1.414928913116455, -0.27153509855270386, 1.1543118953704834, -0.3530850112438202, -0.2695389688014984, 1.778451681137085, -0.8954054117202759, 1.6981934309005737, 0.16413472592830658, -0.159726083278656, 0.6916550397872925, 0.7230358719825745, 0.5159517526626587, 0.3497103154659271, 0.9014212489128113, -0.7038158178329468, -0.9310581684112549, -1.441116213798523, 1.0742992162704468, -0.14930422604084015, -0.82489013671875, 0.7568499445915222, 0.9877826571464539, -1.1623948812484741, 1.2394276857376099, 0.6700858473777771, 0.11002866923809052, 0.9039589166641235, -1.0836604833602905, 0.7989481091499329, -0.4494099020957947, 0.7603596448898315, -0.03175311163067818, -1.5253580808639526, -0.34138089418411255, 1.1291379928588867, 0.3226649761199951, 1.1429344415664673, -0.9496543407440186, 0.8073541522026062, -0.9889289736747742, 0.05072089284658432, 0.2978542447090149, -1.345790147781372, -0.535496711730957, 1.2082650661468506, -0.7938873171806335, 2.7366344928741455, -1.3060566186904907, 1.8929270505905151, -0.18464817106723785, 0.47250762581825256, 0.022044293582439423, -0.3868838846683502, 0.3357113301753998, -0.28333374857902527, 0.9563019871711731, -0.5253310799598694, -0.3652828335762024, -0.41641244292259216, -0.043572187423706055, 0.4055599570274353, -0.058652300387620926, -0.9587429761886597, 0.6643285155296326, -0.7545256614685059, -2.1447906494140625, 0.6490417122840881, 0.21109263598918915, -0.17364245653152466, -0.4939455986022949, 0.7704200148582458, 1.0925573110580444, 0.5978507399559021, 0.9430162310600281, 1.2043074369430542, 1.3318705558776855, -1.4447828531265259, 0.7830952405929565, -0.6352750062942505, 2.062635660171509, -0.6223004460334778, -0.7861931324005127, -0.27449873089790344, 0.18950198590755463, -1.590147852897644, -0.2519400417804718, 0.0054326606914401054, -0.33157050609588623, -0.19666238129138947, -0.44396165013313293, 1.948871374130249, -1.0001485347747803, -0.5303198099136353, 0.0133686987683177, 1.2719699144363403, 0.15412195026874542, -0.5505682229995728, -0.7470611333847046, 1.1536331176757812, -0.37790700793266296, -0.45013731718063354, 0.9513214826583862, 1.5913034677505493, 1.5733253955841064, 0.6813097596168518, -1.7091500759124756, 0.7686493396759033, -0.415693998336792, -1.0921393632888794, 0.6259481906890869, 0.45023196935653687, 0.46368324756622314, 1.0856051445007324, 0.3846498131752014, 0.007303131278604269, -1.768617033958435, 1.392442226409912, -0.7126631736755371, -2.6034696102142334, 1.0152238607406616, 0.720101535320282, 0.004393913317471743, 0.04596586152911186, 0.6319348216056824, -2.0901544094085693, -0.28292927145957947, 0.4144599139690399, -0.4735242426395416, 1.104190468788147, -1.2650386095046997, -0.9551916122436523, -0.5042459964752197, -0.23297041654586792, -0.41864654421806335, -1.1359200477600098, 0.30735597014427185, 1.3923499584197998, -1.6106280088424683, 0.8292458653450012, 0.13350048661231995, 0.26514995098114014, -1.0439850091934204, -0.3219979405403137, -0.49923574924468994, -1.1718788146972656, -0.39170581102371216, 1.7840955257415771, 1.5931196212768555, -0.6532514691352844, 1.6536149978637695, 0.004048828035593033, 0.8573157787322998, 1.9633272886276245, -0.5680276155471802, 0.61830073595047, -1.5542420148849487, -0.03932030126452446, 0.2602384388446808, -0.5367861390113831, 1.556477427482605, 1.3419189453125, -0.8556420803070068, -0.06501726806163788, -0.6267438530921936, -1.1335607767105103, 0.94753098487854, -2.0547282695770264, 0.2996141314506531, -0.46682867407798767, -1.7001961469650269, -0.7142133116722107, 0.5297958254814148, -1.1247479915618896, -1.17986261844635, -1.1335618495941162, -1.1252120733261108, 0.24595330655574799, -1.0613110065460205, -1.8040715456008911, 1.3447493314743042, 0.8457071781158447, 0.06718592345714569, -1.8107719421386719, 1.0168737173080444, 0.44253164529800415, 0.2341011017560959, 0.5258339643478394, -1.0540703535079956, 0.4833202064037323, 0.9253205060958862, -1.4840734004974365, 0.9421958327293396, -0.2297854870557785, 0.8858128190040588, -0.9463790655136108, 1.2274845838546753, -1.5809065103530884, 2.0000367164611816, 0.48387622833251953, -0.3302038609981537, -0.3947296738624573, -0.9297382235527039, -0.7666171789169312, -0.7101801633834839, -0.15915998816490173, -1.582554817199707, -0.43174654245376587, -0.3104958236217499, 0.4661458432674408, 0.5944167375564575, 0.6391139030456543, -0.0612153559923172, 1.0024968385696411, -0.9831819534301758, -0.2996920645236969, 1.4419174194335938, -0.5773148536682129, 0.48498761653900146, -1.3125214576721191, -1.4112918376922607, -0.7512736320495605, 0.7916213274002075, -1.7774076461791992, -0.16979995369911194, 0.714738667011261, -0.22392535209655762, 0.6528580188751221, 0.5426933169364929, 1.6524964570999146, 0.7682845592498779, 1.4903556108474731, 0.46330952644348145, -0.7261303067207336, 1.081769347190857, 1.0508699417114258], [0.3756031394004822, -0.027863366529345512, -0.8444697260856628, -1.0614643096923828, -2.3050713539123535, 0.7574873566627502, 0.6902563571929932, -0.279460608959198, -0.52561354637146, -0.23467369377613068, 0.4308791160583496, -0.4779702126979828, -0.2032262235879898, -0.709994912147522, 0.9847375154495239, 0.09406781196594238, -1.1201958656311035, -0.6792160868644714, 1.423495888710022, 2.625727415084839, 0.34703266620635986, -0.03551019728183746, 0.08560541272163391, -1.2934308052062988, -1.0882643461227417, -0.09336568415164948, 1.732803225517273, 0.4110763669013977, -1.8126295804977417, 0.9813922643661499, 0.4844087064266205, -1.0127023458480835, -0.8835897445678711, 0.25573188066482544, -0.4931115508079529, 0.518568217754364, -0.5206899642944336, 0.2242782711982727, 0.6446083784103394, -1.0059409141540527, 0.05605100095272064, 0.9244903922080994, 0.8745472431182861, -0.08632976561784744, -1.4554990530014038, -0.9519277811050415, 0.9861484169960022, -1.9407504796981812, -1.2504730224609375, 0.5026542544364929, -0.18828409910202026, -1.204529881477356, -1.3787094354629517, 0.11477446556091309, -0.6496912837028503, -0.8532248139381409, 0.438778281211853, -1.2515379190444946, -0.6345775723457336, 1.7925488948822021, 1.5439822673797607, -0.2762436866760254, -1.0535708665847778, 0.2678144872188568, 0.40220630168914795, -0.7843870520591736, 0.5068188309669495, 0.9932721257209778, -0.33771976828575134, 1.5552842617034912, -0.11921780556440353, -0.1861126720905304, 1.1806442737579346, -0.4893086850643158, 0.7934892773628235, -0.011073981411755085, -0.7565661668777466, 0.9433028697967529, -0.9481620788574219, -1.0157828330993652, 1.220854640007019, 0.9069101810455322, -0.13435140252113342, 1.5498119592666626, -1.2557132244110107, 0.27207520604133606, -1.0660918951034546, 1.2057762145996094, 2.5910000801086426, -0.14252141118049622, 0.15425419807434082, -0.19709575176239014, 2.0477700233459473, -1.2051963806152344, -0.4260108470916748, 2.340291738510132, -1.1978706121444702, -0.6303122043609619, 1.316001057624817, 0.9722931385040283, -0.128468319773674, -0.9089916348457336, 1.0840959548950195, 1.6671456098556519, -0.8453512191772461, -0.4145159423351288, -1.1447683572769165, 0.4254724383354187, 1.0357768535614014, 0.4799898862838745, -1.1872341632843018, 0.7009308338165283, -1.6851317882537842, -0.2320171445608139, -0.655773937702179, -0.18301880359649658, 0.14290624856948853, -1.2790727615356445, 1.4277766942977905, 0.5718206763267517, 0.8142169713973999, 1.5016769170761108, -0.7259582281112671, 0.2712491750717163, -1.5975375175476074, -0.35932502150535583, -0.6681226491928101, 0.3267848789691925, -0.7417354583740234, 0.6271671652793884, 1.21538507938385, -0.4372101128101349, 0.4356766641139984, -1.102795124053955, -0.15958158671855927, 1.4831092357635498, 0.8877318501472473, 0.08734957873821259, -0.013673423789441586, 1.1505308151245117, 0.3069875240325928, -1.243111491203308, -0.6383619904518127, -0.9696663022041321, -0.1766369342803955, -0.3789108395576477, 0.3597090244293213, -1.3447998762130737, 0.0873372033238411, 0.1644907146692276, 0.7707486152648926, -0.11562156677246094, 0.7492980360984802, 0.6946107149124146, 0.8705319762229919, -1.1880755424499512, -2.0934953689575195, -2.2261195182800293, -0.40706637501716614, -0.6861399412155151, -1.0087536573410034, -0.28743186593055725, -0.7140011787414551, -0.37676188349723816, -1.1014792919158936, -0.769602358341217, -0.6563646197319031, -0.6852397322654724, 1.3700478076934814, -2.3045365810394287, -1.2229541540145874, -1.2032678127288818, -1.2961370944976807, 0.3345966637134552, 0.04344068840146065, -1.0573402643203735, -0.9861172437667847, 0.2827070355415344, -0.4658869206905365, -0.46596240997314453, 1.107648491859436, -1.2232227325439453, 0.7412775158882141, 0.07883995771408081, -0.6699275374412537, -0.8361235857009888, -0.6765239238739014, 0.7210121154785156, 0.28901219367980957, 0.5276286005973816, -0.9518516063690186, 0.06373360753059387, 0.2742026150226593, -0.9634115695953369, 0.765509843826294, 1.2637639045715332, -0.39900848269462585, 0.9771851897239685, 0.706642210483551, 0.5865576863288879, 0.06558063626289368, 0.5629025101661682, -1.2922321557998657, -0.13747017085552216, 0.058968525379896164, 1.243111491203308, -0.14721402525901794, -0.3683774173259735, 0.3932904899120331, -2.3840177059173584, 0.6176625490188599, -1.3043761253356934, -0.29403918981552124, -0.7031337022781372, -0.060791853815317154, -0.1282503306865692, 0.5131250023841858, -0.04762682318687439, -0.6186171174049377, -0.9277941584587097, -1.4012699127197266, -0.8744435906410217, 1.6542491912841797, -1.0742998123168945, -0.8862929344177246, -0.44181832671165466, 1.0863046646118164, -0.36742672324180603, -1.3755357265472412, -0.8310760855674744, 0.09283090382814407, -0.14910101890563965, 0.7465556859970093, -0.7781239748001099, -0.8187121748924255, 1.2700746059417725, 1.4043660163879395, 0.7221945524215698, 0.27731823921203613, 0.9141436815261841, -2.5851869583129883, -1.3971542119979858, 1.6372082233428955, 0.000771299353800714, -0.15641789138317108, 0.6991986632347107, -0.2766835689544678, 0.8874754905700684, -0.37749090790748596, 1.7683086395263672, 1.2816708087921143, 1.9521173238754272, 1.6095908880233765, -0.7387606501579285, 0.5660629868507385, 0.1739882379770279, 0.009805308654904366, -0.41165173053741455, 0.7883722186088562, 0.6348147392272949, -0.253596693277359, -0.6444162726402283, 0.4439314007759094, 0.5642877221107483, -1.1577094793319702, 0.3089113235473633, 0.9557716250419617, -0.3347512185573578, 0.1999620497226715, 0.38148531317710876, -1.7484619617462158, -0.8151628375053406, 0.7960976958274841, 0.0022608067374676466, -1.6985212564468384, 0.03692856431007385, 0.45712897181510925, -0.277334988117218, 0.12153010070323944, -0.22059717774391174, -0.7165625095367432, 0.8156227469444275, 1.6028231382369995, -0.6911250352859497, -0.6062650680541992, 0.6919968128204346, -0.07285196334123611, 2.266651153564453, -0.5749267339706421, 1.4437909126281738, -0.05683540552854538, -1.4589362144470215, -0.05603737384080887, -0.18972453474998474, 1.9701532125473022, 0.470232218503952, 1.7594975233078003, -1.3358343839645386, 0.8573722839355469, 1.5219752788543701, -1.1216858625411987, -1.6771436929702759, -0.24910308420658112, -1.346733808517456, -0.23266737163066864, -0.9793170690536499, 1.2380297183990479, -0.4622678756713867, -0.44092532992362976, 0.2309926450252533, -0.14380612969398499, 1.3671882152557373, 1.8246831893920898, 0.9543720483779907, 0.18612128496170044, 0.45129162073135376, 0.4832627475261688, 1.580232858657837, -0.26559531688690186, -0.2070690542459488, -1.616267204284668, -0.9199200868606567, -1.8981891870498657, -0.008840260095894337, -0.6440680027008057, 1.5413086414337158, 0.5186347365379333, 0.07970469444990158, -0.7225147485733032, -1.5454509258270264, 0.41297218203544617, 0.7876338362693787, 0.21912607550621033, 0.8861010670661926, 1.4326019287109375, -0.26300570368766785, 1.5381044149398804, 0.023233307525515556, 0.8715749979019165, -2.0323338508605957, 1.8940714597702026, 0.2439049333333969, 0.32718250155448914, 0.6698198914527893, 2.456743001937866, -0.7091341614723206, -0.35283467173576355, 0.8061543107032776, 0.8489080667495728, 0.20403465628623962, 1.335463285446167, 2.127955675125122, 1.289936900138855, -0.07485532015562057, 1.1848968267440796, -1.8683058023452759, -1.6416751146316528, 0.9742658138275146, 1.0193686485290527, 1.9771497249603271, -0.7212304472923279, 0.029905859380960464, -0.6064772009849548, 0.8176712989807129, -1.5800529718399048, 1.0482879877090454, -0.7200782895088196, 0.3315264582633972, -0.2855766713619232, 0.20543956756591797, -1.0373384952545166, -0.9684579372406006, 0.29046630859375, -0.021028606221079826, -1.8372383117675781, 0.4402196705341339, 1.7613462209701538, 0.029635164886713028, -1.0198789834976196, 0.4655776023864746, 0.8363083600997925, -1.7994433641433716, -0.7064640522003174, 0.6802123188972473, -0.18266838788986206, -0.9339067339897156, 1.0113928318023682, -0.24958746135234833, 0.2735464572906494, -1.2000879049301147, -0.10811373591423035, -0.682175874710083, 1.4154335260391235, -0.3598922789096832, 0.9349469542503357, -0.7276556491851807, -1.631453514099121, 1.723668098449707, 0.9186555743217468, 0.10504721850156784, -1.6871753931045532, 1.1653074026107788, -0.7387977838516235, -0.0008515436202287674, 0.037499185651540756, 1.5540162324905396, -0.38538581132888794, 0.799263060092926, 0.04809364676475525, -0.14233900606632233, -1.5741904973983765, -1.8623489141464233, -1.0010706186294556, -0.3345707356929779, -0.7315247654914856, 0.15940046310424805, 0.16823576390743256, -0.25252094864845276, 1.2484054565429688, -0.15920762717723846, -1.9178106784820557, 0.8256532549858093, -0.6000785231590271, -0.8458364605903625, 1.2010960578918457, 0.21054404973983765, -0.5700305700302124, 1.8938002586364746, -0.6660618782043457, -0.04166834056377411, 1.2447372674942017, -1.077728509902954, 0.7057023644447327, 1.2032216787338257, -0.708352267742157, -0.012064267881214619, -0.2282036393880844, 1.8401929140090942, 0.8131833076477051, -2.0970208644866943, -0.28761059045791626, 0.49037760496139526, 1.027567982673645, -0.7089574933052063, 0.04434555396437645, -0.7087898850440979, 1.7829431295394897, 0.5390859842300415, 0.34494489431381226, 1.6285309791564941, 1.3021389245986938, 0.8510181903839111, 1.0576374530792236, 1.842218279838562, -0.3686893582344055, -0.6312329769134521, 0.3339179456233978, 0.9135072827339172, -1.846871018409729, -1.6276124715805054, -0.7067034840583801, -1.7768875360488892, -1.7139664888381958, -0.547225832939148, -0.6811439394950867, -0.5369576811790466, 0.06702370941638947, 3.498589038848877, 0.7094154357910156, 1.212953805923462, 0.26895108819007874, 0.5502559542655945, 0.0041682785376906395, -0.7486990094184875, -0.008344960398972034, -1.4139693975448608, 0.5164469480514526, 1.6184356212615967, 0.9072204232215881, -1.6663931608200073, 1.2065478563308716, 1.125200629234314, -0.8762925863265991, -1.526398777961731, 2.917977809906006, -0.6656228303909302, -0.5831631422042847, 0.9787949919700623, 0.17334336042404175, -0.40520501136779785, 0.28170838952064514, -0.6561905145645142, -0.09049079567193985, -0.4597376883029938, 1.865993618965149, -1.051517128944397, -1.2907711267471313, 1.3724254369735718, -0.6636505126953125, -0.5651344060897827, -0.19625228643417358, -2.0833394527435303, 0.540404200553894, 0.8222767114639282, 0.6737590432167053, 0.785533607006073, 0.0036340279038995504, 0.10349264740943909, -0.8513302803039551, -2.1119155883789062, -0.935397744178772, -0.4669409394264221, -1.1995779275894165, 0.6923551559448242, 0.8118393421173096, -0.4444950819015503, 1.0719283819198608, -0.9058558940887451, 0.7651999592781067, -2.2714626789093018, -1.1887481212615967, 0.6422713994979858, -1.3299436569213867, -1.4004467725753784, -0.1264587640762329, -1.4070971012115479, 2.1093969345092773, 1.246283769607544, -0.2661972641944885, 0.3617232143878937, -0.5699517726898193, -1.4033681154251099, -0.6423566341400146, 0.08964918553829193, 0.7481634616851807, -1.432479739189148, -0.2053012102842331, 0.2547774314880371, 0.7170149087905884, 0.8854183554649353, 0.6480691432952881, 0.5442417860031128, -1.6032711267471313, 2.3550772666931152, 0.5882859826087952, -0.8899075388908386, 1.062604308128357, 1.1686612367630005, -0.643865704536438, -1.002199649810791, 0.7871174812316895, -0.06806392967700958, 0.7446821928024292, 2.3732974529266357, 0.8702172636985779, 0.45334720611572266, -2.2673370838165283, 0.10836378484964371, 1.2341471910476685, -0.49534788727760315, 1.9506306648254395, 0.23821479082107544, 0.22841614484786987, -0.018741218373179436, -1.017808437347412, 0.37749502062797546, 1.1273281574249268, 1.1531708240509033, -0.27041763067245483, -0.11802040040493011, -0.2402084618806839, 2.135289192199707, 0.7973963618278503, -1.2668148279190063, -0.4975709915161133, -2.4239184856414795, 0.5508322715759277, 1.056943416595459, -0.29707545042037964, 1.6645700931549072, -0.6417523622512817, 0.6924940347671509, 1.4439489841461182, -1.1002342700958252, -1.0832608938217163, 0.6066411733627319, -0.43919792771339417, 0.1210469976067543, 2.2419464588165283, 1.8707523345947266, 1.467463493347168, -0.09849860519170761, -1.0652192831039429, -0.47962963581085205, -0.4827203154563904, -1.3641544580459595, 0.617810845375061, 1.9341483116149902, 0.015273754484951496, -0.09696453809738159, 0.9175695180892944, 0.4770429730415344, -0.5450717806816101, -0.5732187032699585, 0.25290343165397644, 0.2817259132862091, -0.9773333668708801, 0.21634118258953094, 1.0349187850952148, -0.006313859485089779, 1.6221342086791992, -1.3924843072891235, 0.6100396513938904, -0.16492746770381927, 0.33321353793144226, 1.4586658477783203, 0.11655372381210327, 0.13901911675930023, -0.37740400433540344, -0.32925984263420105, -0.5721573233604431, 0.2941429316997528, -1.1690417528152466, 0.6043908596038818, -0.938357412815094, 0.4914289712905884, -0.3260362446308136, 0.4620741903781891, -2.1745083332061768, -0.9429587721824646, 1.0578653812408447, 1.293999195098877, -0.1661066859960556, -0.042843692004680634, 0.7192047834396362, 0.8265385031700134, 0.3254605531692505, -0.46278294920921326, -0.40628284215927124, 0.3701573610305786, 1.8014169931411743, 0.8191498517990112, 1.5597599744796753, -0.01978711597621441, -0.5198946595191956, 0.890377938747406, 1.9148473739624023, -1.21087646484375, -1.877533197402954, -1.6963390111923218, -0.7051795125007629, -0.9352830648422241, 0.2288568913936615, -0.5275030136108398, -1.4364590644836426, 0.6746947169303894, -2.035266876220703, 0.3700675964355469, 0.27804362773895264, 1.0410162210464478, 0.8395813703536987, 0.6007423400878906, 1.5091136693954468, -0.029890984296798706, -1.4210132360458374, 0.0336020365357399, -1.4666475057601929, 0.804034948348999, -0.947041928768158, -0.8840227127075195, 1.1817795038223267, -0.9215547442436218, 0.48782864212989807, -0.12668752670288086, -0.2406076192855835, -1.498228907585144, -0.7774523496627808, 1.2209992408752441, 0.14575473964214325, -1.167816400527954, -2.0416414737701416, 0.0832059308886528, 1.4762084484100342, 1.6567236185073853, -0.1293702870607376, -0.2310570478439331, 0.47706320881843567, 1.888312816619873, -1.4462741613388062, -0.1701822131872177, -1.1893017292022705, 0.2646704316139221, 0.06939490884542465, 0.12289692461490631, 1.329017996788025, 0.1571321040391922, 1.6546471118927002, -0.954551637172699, 0.38939914107322693, 0.6858370900154114, -0.23514999449253082, -0.18524597585201263, 0.1197749525308609, 1.332700490951538, 2.0741147994995117, 1.0464847087860107, 0.06902012228965759, 0.46820640563964844, 0.7559083700180054, 0.5486436486244202, -0.04272998496890068, -0.7583131194114685, 0.73338383436203, -2.2935235500335693, -1.3512048721313477, -0.6278165578842163, -0.7615063190460205, -0.7759857773780823, -0.592985987663269, 0.5671965479850769, 2.3405489921569824, -0.29005104303359985, 0.2560752034187317, -0.20579694211483002, 1.1452723741531372, 1.5443254709243774, 0.32707592844963074, 1.429503083229065, -0.23087579011917114, -0.17362672090530396, -1.7533390522003174, -2.5305657386779785, -0.6233307123184204, 0.041745275259017944, 0.3643641769886017, 0.8216779828071594, 1.3568627834320068, 0.11335981637239456, 0.4607512354850769, -0.062122784554958344, -1.0093841552734375, 0.3126140832901001, 1.9155608415603638, -0.5903050899505615, -0.4127461016178131, 0.3736754059791565, -1.9967018365859985, 1.3045556545257568, -2.154207229614258, -0.9019568562507629, -0.8116832971572876, 0.4760984182357788, 1.5623135566711426, 1.3141506910324097, 0.16555270552635193, -1.4906954765319824, -0.0027868186589330435, 1.6842771768569946, 0.5829586982727051, -0.16695865988731384, -0.16382023692131042, -0.5568339824676514, -1.0082253217697144, 0.5237802863121033, 0.8648101091384888, -1.2581191062927246, -1.9942148923873901, 0.34183233976364136, 0.4643819332122803, 1.6375150680541992, -1.610095739364624, 0.9481003880500793, 1.975574254989624, -1.069727897644043, -0.9308551549911499, 0.3121512830257416, 0.7549769878387451, 1.0287941694259644, 0.10331422090530396, 1.0235378742218018, 0.34581926465034485, 0.36351078748703003, 1.1467584371566772, -0.5076205730438232, -0.6317179203033447, -0.4662138819694519, -0.29644775390625, -0.8829301595687866, 0.6427538394927979, 0.1293470710515976, -0.6853217482566833, -1.9347343444824219, -0.3541181683540344, 0.18992216885089874, 0.5017955303192139, -0.6289186477661133, 1.4553849697113037, -2.1071698665618896, 2.0580801963806152, 0.8674045205116272, -0.25077927112579346, 0.7920243144035339, -0.5073845982551575, 0.23057298362255096, -0.444096177816391, 0.7700060606002808, -0.42875316739082336, -0.0749436691403389, 0.9172049164772034, -0.7579859495162964, 0.2779645621776581, -1.1522122621536255, 1.1929844617843628, 0.05796751752495766, 0.9877950549125671, -1.3687623739242554, -2.1286189556121826, 0.2895289957523346, -0.32382380962371826, -1.1047608852386475, 0.7428566813468933, -0.7580277919769287, 0.2949521541595459, -0.5226787328720093, -0.6735244393348694, 0.9707385897636414, 0.6234074234962463, -1.5368449687957764, -0.847736656665802, 1.6713192462921143, -2.2604920864105225, -1.0472235679626465, 0.7800152897834778, -0.24916259944438934, 0.9234222173690796, -0.6290728449821472, 0.14082781970500946, 0.31310155987739563, 0.0966717004776001, -0.7040256261825562, -0.7837455868721008, -1.080422282218933, 1.744977355003357, 1.049897313117981, -0.15545356273651123, -0.4805193245410919, 0.26546090841293335, 0.9693335294723511, -1.1564795970916748, -0.7118484973907471, 0.6066960692405701, -1.1844364404678345, 0.6626935005187988, 0.8184782862663269, 0.0655670166015625, -0.5433285236358643, -0.9704286456108093, -1.0441876649856567, -0.10363244265317917, 0.7232168316841125, 0.24717935919761658, 0.9286932349205017, -1.1150288581848145, 0.0015873019583523273, 0.5016255974769592, -0.38608402013778687, -0.14129787683486938, 1.1604818105697632, -0.44804951548576355, 1.0782724618911743, 0.4809616208076477, -1.375040888786316, 0.3921256959438324, -0.5626858472824097, 0.7380154132843018, -1.0223984718322754, 1.1826751232147217, -0.11973186582326889, 0.8965205550193787, 1.0564719438552856, 1.0697863101959229, -0.25457680225372314, -0.6212725639343262, -0.2631765305995941, -2.136256694793701, -2.069756031036377, 1.9396851062774658, -1.7516582012176514, 0.1299835443496704, 0.6247512698173523, -0.6602136492729187, -1.3179694414138794, 0.9537203311920166, -0.2931918203830719, -0.6032767295837402, -0.19840021431446075, -1.2478574514389038, 0.36018162965774536, 0.21803273260593414, -1.3814218044281006, -1.1400960683822632, 0.5987185835838318, 0.5712341666221619, -0.7189531922340393, -1.6369246244430542, -0.1628805249929428, -0.4087044298648834, -1.4120796918869019, 1.2334229946136475, 0.04779261723160744, 1.0625035762786865, -0.8148303031921387, 1.0278332233428955, 0.9834636449813843, 1.8310246467590332, 0.41054636240005493, -0.35432446002960205, -0.808090090751648, 1.277113676071167, -1.8304047584533691, 0.7338385581970215, 1.2060693502426147, -1.1546965837478638, 0.8658158779144287, -0.41135141253471375, -2.3086178302764893, -0.571281373500824, 1.2719223499298096, -0.4147597551345825, 1.403839111328125, 0.5121445655822754, -0.3585449457168579, 0.12721098959445953, -0.05863645300269127, -2.4661715030670166, 0.028683587908744812, 2.2284328937530518, 0.381233811378479, -0.5008121728897095, 0.7557650208473206, -0.5899785161018372, 1.5888878107070923, 0.12002215534448624, 0.28882724046707153, 0.14389771223068237, 0.07046731561422348, 0.3993859887123108, 0.050252750515937805, -1.7895146608352661, -0.6292912364006042, -1.0462965965270996, 0.17693836987018585, 0.0731620192527771, 0.5688341856002808, 0.27789679169654846, 2.279296398162842, 0.3628956079483032, -0.683860182762146, 0.200193852186203, -1.6416093111038208, 0.7055835723876953, -0.17163561284542084, -0.5939338803291321, -0.29921746253967285, 1.285671591758728, -1.524253249168396, 1.705730676651001, -0.0593925416469574, 0.6444798111915588, 0.4529489278793335, 0.5321204662322998, 0.11553829163312912, -0.6577698588371277, 0.9208409190177917, 0.33259591460227966, 1.3885722160339355, -0.6078072190284729, -1.6711909770965576, -1.0423308610916138, 1.2192299365997314, 0.13157829642295837, 0.5676803588867188, -1.8288369178771973, -1.2071638107299805, 1.041098952293396, 0.09817174822092056, -0.04572978988289833, -0.28751644492149353, 0.2904109060764313, 1.0932978391647339, -0.414917528629303, 0.193497434258461, 0.3580509424209595, 1.7966299057006836, -0.8376523852348328, 0.9513962864875793, -0.34707939624786377, -0.0965547040104866, 0.5253930687904358, -1.004308819770813, -0.9701696038246155, -0.01680365391075611, 0.6678023934364319, 0.7469049692153931, 1.510422945022583, 0.0970507562160492, 0.9062001705169678, 0.17659012973308563, 0.8684178590774536, 0.5990306735038757, -0.9392346739768982, 0.5549547672271729, 1.6423109769821167, -0.8417218923568726, -1.2933568954467773, 0.5452159643173218, -0.19828204810619354, 0.47443103790283203, 0.6090948581695557, 0.05915546789765358, 0.11689039319753647, 0.8898818492889404, -1.2813166379928589, 0.6380922198295593], [0.016205012798309326, 0.058343786746263504, 0.17694643139839172, -0.1282602846622467, -1.9846737384796143, 0.005104639567434788, 0.22152379155158997, 0.9299262166023254, 0.23595765233039856, 2.0412557125091553, 0.8334957957267761, 1.8305853605270386, 0.5874305963516235, -0.769763708114624, 0.4550054669380188, 0.6238031387329102, -0.44541746377944946, -0.1444733440876007, 2.5401740074157715, -0.01891220547258854, 0.30714717507362366, -1.0059754848480225, 0.20974382758140564, 0.3507344722747803, -0.35207292437553406, -0.015024499967694283, 1.9388933181762695, -0.32213255763053894, -0.8286299705505371, 0.16787661612033844, 0.6121090054512024, 0.15614855289459229, -0.9379175305366516, -0.688632071018219, 1.1859984397888184, 0.4975593090057373, -0.2378697693347931, -0.4133293926715851, 2.0892598628997803, -1.9038199186325073, 1.4303220510482788, -1.9491205215454102, 2.579254388809204, -0.5900334715843201, -0.33289864659309387, 1.0928043127059937, 1.4624054431915283, 1.3237637281417847, -1.1302454471588135, 1.4832656383514404, -0.08481333404779434, -1.065986156463623, -0.8076382279396057, 0.5911298990249634, -0.6997077465057373, 0.4407396614551544, 0.6243937015533447, -0.544701337814331, -0.6981532573699951, -1.3176424503326416, 0.18751217424869537, -0.12112336605787277, -0.8586322069168091, 0.5003514885902405, -0.32478973269462585, -0.5626768469810486, -1.0467034578323364, 1.9909876585006714, -1.343697428703308, -0.6017765402793884, -2.106938123703003, -0.8974789381027222, 0.25940898060798645, 0.912767231464386, -2.272486686706543, -0.5643040537834167, 0.02328469790518284, -0.14772818982601166, -2.0803260803222656, 1.3065353631973267, 1.3232077360153198, 0.34375327825546265, 0.8797450065612793, 1.4271513223648071, 0.7937297821044922, 0.01898062229156494, -1.1547733545303345, -0.40333738923072815, -0.5441408753395081, -1.4278291463851929, 1.5319898128509521, 1.3456507921218872, 0.62535160779953, 0.32292601466178894, 0.5934954285621643, -0.6967156529426575, 1.0669307708740234, -0.7696672081947327, -0.6150006055831909, -0.21115665137767792, 0.8507424592971802, -0.691224217414856, 0.8931222558021545, 0.8239266872406006, -1.1543300151824951, -0.1619315892457962, 0.8858339190483093, 0.5660507678985596, 0.3414751887321472, -0.2563006281852722, 0.3060356676578522, 1.650388240814209, -2.8219830989837646, -0.14416399598121643, 0.33048340678215027, 0.04951995611190796, -1.4278457164764404, -1.6718536615371704, -0.5043854713439941, 1.1858139038085938, 0.4514811933040619, 0.37264636158943176, 0.20406995713710785, -1.9874608516693115, 0.2521110475063324, -1.6289761066436768, 0.507182776927948, -0.46053674817085266, -0.46987301111221313, 0.07053042203187943, -2.2243645191192627, -1.101487636566162, 0.5284603834152222, 2.8999698162078857, 0.2735927104949951, 0.30690354108810425, 1.5383297204971313, -0.16310825943946838, 0.9634690284729004, -0.5331194996833801, 0.4657883644104004, -1.1432380676269531, 0.8867841958999634, 1.6429134607315063, 1.3426804542541504, 0.24091264605522156, 0.8801479339599609, 0.17446637153625488, 0.21297301352024078, -0.1535005420446396, 0.4042666256427765, 2.055898427963257, 0.0930696427822113, -0.5458388328552246, -0.1845291405916214, 0.49911606311798096, 0.879412055015564, 0.5414221882820129, 1.2055422067642212, -0.5986685752868652, 0.4331054985523224, 0.5968075394630432, -0.6513312458992004, -1.0512667894363403, -0.8118459582328796, 0.9591275453567505, 0.8838333487510681, -0.8554003834724426, 1.4352251291275024, -1.841710090637207, 0.3340243995189667, -0.3285468816757202, -0.572960615158081, -0.5082053542137146, -0.08739043027162552, -1.1378546953201294, -2.5743722915649414, -0.7561966776847839, 0.1668267548084259, 0.8205682635307312, -1.4677929878234863, 0.3333016037940979, -2.1196353435516357, 0.7200750708580017, 0.525397002696991, 1.111383080482483, 0.6480945348739624, 0.9597601890563965, -0.8833404183387756, 1.2484456300735474, 1.4615992307662964, -0.5771433115005493, 2.0090653896331787, 0.8686721324920654, -0.23813804984092712, -1.1569629907608032, 0.8604285717010498, 0.16752316057682037, -0.20458674430847168, 0.5153828263282776, -0.057992320507764816, -2.1987035274505615, -0.089567169547081, 1.3289520740509033, 1.1916316747665405, -0.12650679051876068, 0.3795705735683441, -0.27513378858566284, -0.9630445837974548, 0.014104997739195824, -0.2885225713253021, -0.3485068082809448, -0.6727679371833801, 0.8709625601768494, -0.2449924349784851, 0.38757768273353577, -0.23507049679756165, 1.0446516275405884, 0.1809760481119156, -0.9190725684165955, 0.6448739767074585, -0.9893907308578491, -1.0304973125457764, -0.5162053108215332, 0.3020594120025635, 0.1733611673116684, -0.37292787432670593, 0.5868886113166809, 0.16732458770275116, 0.5842851996421814, -0.45206865668296814, 0.6268380284309387, 0.9883177876472473, -0.1195734515786171, 0.5196679830551147, -1.0820066928863525, -0.1458875983953476, -0.48071736097335815, -1.693786859512329, -1.8055356740951538, -0.1590195596218109, 1.387691617012024, 0.24833141267299652, 1.3210371732711792, 0.9253331422805786, 1.8050295114517212, 0.09734232723712921, 0.797579288482666, -1.488355040550232, -0.3458016514778137, -0.6146154403686523, 1.2627286911010742, -0.9658374190330505, 0.6456294655799866, 1.054787516593933, 0.4794030785560608, 1.5105994939804077, 1.519449234008789, -1.1959307193756104, 0.9366863965988159, -1.4651525020599365, 0.15277248620986938, -0.33980420231819153, 0.41378721594810486, 1.2969443798065186, 0.4386942982673645, -1.8579556941986084, 0.7000474333763123, -1.2722117900848389, -0.2967202663421631, 0.46623650193214417, -1.7327789068222046, 0.6687374114990234, -0.852528989315033, 1.194762110710144, 0.03446606174111366, -0.41148844361305237, -0.6095248460769653, -0.3438967168331146, 0.9648995399475098, 1.3621339797973633, 1.3381301164627075, 0.37653598189353943, 0.4793683886528015, 0.11660675704479218, -0.5876260995864868, -2.681912660598755, 0.06610997021198273, -0.3018849790096283, 0.990138053894043, -0.9032738208770752, 1.5845919847488403, -0.5010770559310913, -2.371051788330078, -0.5591832995414734, -1.0023086071014404, -0.06357669085264206, -0.7320056557655334, 0.9446505904197693, 0.5786828994750977, -0.5102505087852478, 0.8157797455787659, 0.14280174672603607, 0.8444566130638123, -0.37533286213874817, 1.1069098711013794, -0.08300144225358963, -0.9435243606567383, -0.3784582018852234, -1.0682425498962402, 1.7891064882278442, -0.5379346609115601, 0.6637943983078003, 0.3049178123474121, -0.0905819684267044, -0.063285693526268, 0.51225745677948, -2.7765579223632812, 0.8366311192512512, 0.910292387008667, -0.15176653861999512, -0.6175075769424438, 2.812455415725708, -0.6763740181922913, 1.0750398635864258, 0.7374628782272339, -0.7807661294937134, 0.29185929894447327, 0.9259344339370728, 1.2172214984893799, 0.10724075138568878, -0.6893683671951294, 0.08684669435024261, -1.12448251247406, -0.06150677800178528, -0.8058432340621948, -0.8067329525947571, -0.4151759743690491, -0.4855765998363495, 0.08874355256557465, -0.5473282933235168, -2.0384926795959473, 1.4466952085494995, 0.03313187509775162, 0.6705582737922668, 1.76308012008667, -1.4575989246368408, -1.5203194618225098, -1.078252911567688, 0.706027090549469, 0.35883015394210815, -0.7983455061912537, 1.9639607667922974, -0.5583397150039673, -1.155416488647461, -0.2801590859889984, -1.2837857007980347, 0.7636159062385559, 1.554458737373352, -1.7497960329055786, -1.0370430946350098, -1.1444470882415771, -0.210890993475914, -1.31196928024292, 1.6822313070297241, -0.9207121133804321, 0.23253415524959564, 0.981872022151947, 0.057335495948791504, 0.10178160667419434, -0.4190758466720581, -0.3769902288913727, 2.1632063388824463, 0.40436115860939026, -0.10189490020275116, 1.9839800596237183, -0.43194013833999634, 0.9151263236999512, 0.37922245264053345, -0.34345823526382446, -0.1349819004535675, -0.15824547410011292, -0.20298317074775696, -0.23783519864082336, -0.7420023679733276, 2.300950765609741, 1.1119378805160522, 0.6611668467521667, 1.2387621402740479, 0.8971573710441589, -0.31446191668510437, 0.6113725900650024, -0.7046459317207336, 0.6301770806312561, 0.09240037947893143, 0.057783178985118866, 0.5683016180992126, -2.8963799476623535, -0.21746696531772614, -0.3929566442966461, 0.16974098980426788, -0.09969442337751389, -0.2510268986225128, -0.4437999725341797, 0.7657710909843445, -0.17780476808547974, 2.1568961143493652, 2.7337472438812256, -0.6222867369651794, -0.833921492099762, -0.653135359287262, 1.188999056816101, -0.4330300986766815, 0.23597468435764313, 2.2655231952667236, 0.37234756350517273, -0.0143973333761096, 1.3829225301742554, -0.17826120555400848, 1.140478491783142, -0.3042759299278259, -0.7233901619911194, -0.07026710361242294, -1.285382628440857, 1.0828238725662231, 0.19738225638866425, 0.6673265099525452, -0.10612112283706665, 0.7485610246658325, -0.016110997647047043, -1.0501377582550049, 0.21667827665805817, 0.23039278388023376, -1.625680685043335, 0.03830370306968689, 1.1780378818511963, 0.1477191299200058, -0.4077152609825134, 0.09465963393449783, -0.6700096726417542, 0.9070748090744019, -0.9355511665344238, -0.02529176138341427, -0.09038371592760086, -1.4507107734680176, -0.19700810313224792, -0.18549466133117676, 0.6167206764221191, -0.1606215387582779, -0.5749491453170776, 0.48978278040885925, 0.986070454120636, -0.34962737560272217, -0.2895093262195587, 0.4673587381839752, 1.4012492895126343, 1.683997392654419, -0.10093532502651215, -1.1396881341934204, -0.324555367231369, 0.7092010974884033, -1.2849547863006592, 0.05784771218895912, -0.43318021297454834, 0.710645318031311, 1.418116569519043, -0.014220483601093292, -0.7444394826889038, 0.11496417969465256, -0.23985476791858673, -0.1327497363090515, 0.7759042382240295, -0.3000706136226654, -1.3508774042129517, -0.623088002204895, 0.5417762398719788, 0.14576353132724762, -0.1538054347038269, -0.8813349008560181, 0.634496808052063, 0.17135347425937653, 0.6623491048812866, 0.43254321813583374, 0.9129338264465332, -1.3350149393081665, -0.8550589084625244, 0.4457951486110687, -0.9444205164909363, 1.7838960886001587, 1.4265782833099365, 1.0216662883758545, 0.3371613919734955, 0.4596116542816162, -1.6018567085266113, -0.06044449657201767, -0.2013198435306549, -0.2598189115524292, -0.5101571083068848, 1.2867921590805054, -0.8884609341621399, 0.9414582848548889, 0.11159742623567581, 1.7635294198989868, 1.0802818536758423, -0.611910343170166, -0.7076707482337952, 0.6733890175819397, -0.033841971307992935, 1.3484448194503784, 1.8488836288452148, -0.8285407423973083, -1.074792742729187, -0.31177955865859985, -0.1885913759469986, -0.845552921295166, 0.5596787333488464, 0.9098498821258545, 0.10383839905261993, -0.3859036862850189, 0.10484076291322708, 1.5753202438354492, 0.07390789687633514, -0.6401863694190979, -0.8776003122329712, -0.3698248565196991, 0.149037167429924, -0.2672760486602783, -0.12702839076519012, -0.5165000557899475, -0.00969316903501749, -0.06306316703557968, 1.8343181610107422, -0.35744431614875793, 0.8003199100494385, 0.8514974117279053, 0.020543718710541725, -0.40327057242393494, -0.0303807333111763, -0.22388333082199097, 0.8968420624732971, -1.871834397315979, -1.17405366897583, 1.4429435729980469, 0.5684641003608704, -0.6190445423126221, 0.5275780558586121, -0.8612604737281799, -0.617212176322937, 0.8382048010826111, 0.23959651589393616, -0.48855939507484436, -0.8305617570877075, -0.32972508668899536, -1.7108557224273682, -0.3897017538547516, -1.1981911659240723, -0.7254275679588318, 0.06857733428478241, -1.6767014265060425, -0.8745330572128296, 0.9812038540840149, -1.3487064838409424, -0.7296072244644165, -1.0835702419281006, 0.8743742108345032, -0.5512446165084839, 0.46656709909439087, -0.3357866406440735, 0.8239613175392151, 1.6340782642364502, 0.0653269812464714, 2.5682525634765625, 1.0212123394012451, -0.4724225103855133, 0.15179921686649323, -0.18106724321842194, -1.4515615701675415, -1.4482685327529907, 0.7411773800849915, 0.6155602335929871, 1.1493760347366333, 0.4359551966190338, -1.602381944656372, 1.2023768424987793, 0.9442627429962158, 0.6534651517868042, -2.186069965362549, -1.6502017974853516, -0.30145150423049927, -0.02102999947965145, -0.07331427931785583, 2.0695157051086426, 1.0048258304595947, -1.7848492860794067, 0.01876935176551342, -0.6003344058990479, 1.0571025609970093, 1.120404601097107, -0.2211030125617981, -0.007060641422867775, -1.3762166500091553, 0.20090541243553162, 1.1107392311096191, -0.46294406056404114, 0.5068450570106506, -0.7892974019050598, -0.20640769600868225, -0.6971220374107361, -0.9331303238868713, -0.21855293214321136, 1.2321171760559082, 1.235650897026062, 0.49129825830459595, 0.5200621485710144, -0.07185729593038559, 0.023656195029616356, -1.4372766017913818, 0.6121827363967896, 0.1613837033510208, 0.7736594676971436, 1.5401142835617065, 0.4920786917209625, -0.05086592584848404, -0.19787375628948212, -0.12470632791519165, 0.9407555460929871, 1.3434395790100098, -0.7715209126472473, -0.5390702486038208, -0.3996029198169708, 1.408528447151184, -1.564353346824646, -1.4297205209732056, 1.3634939193725586, 0.2887779474258423, 0.699550986289978, 0.5757327675819397, 1.9553492069244385, -0.0736551284790039, 0.15191452205181122, 0.7170682549476624, -0.3186741769313812, -0.020558875054121017, 0.5467501282691956, -0.3491787016391754, 1.1298805475234985, 0.9828622341156006, -2.0205044746398926, 1.4860879182815552, -0.8684958219528198, 1.2790544033050537, 2.014254331588745, 0.5635199546813965, -0.1850505918264389, 0.46573641896247864, 0.40367740392684937, -0.06485915184020996, -1.5828603506088257, -2.378598213195801, 0.3492375314235687, 1.2249850034713745, -0.23920653760433197, -0.5500530004501343, 0.5775387287139893, 0.827242910861969, -1.3200706243515015, 0.17316289246082306, 0.9771263003349304, -0.5164177417755127, 1.2306654453277588, 0.8402149677276611, 1.6599111557006836, -1.5131984949111938, -1.1791025400161743, -0.6170862913131714, 0.4889151155948639, 1.1414365768432617, -2.7962701320648193, 0.6171224117279053, -1.0631425380706787, 0.8601120114326477, -0.4619356691837311, 0.1904970109462738, 0.22357767820358276, 0.3278835415840149, 0.9118431806564331, 1.000761866569519, -1.7228525876998901, -1.690818190574646, 0.5242012143135071, 1.7262251377105713, -0.9567815065383911, -1.4080685377120972, 1.9232838153839111, -1.385305643081665, 0.5770508646965027, 1.7245962619781494, -0.7276265025138855, 1.7767163515090942, -0.5992530584335327, -0.11179646104574203, 1.3710269927978516, -3.0753846168518066, 0.6386899352073669, -0.0068882182240486145, -0.6267617344856262, 1.035109519958496, -1.528140902519226, -1.519525408744812, -1.5606961250305176, -0.2840067744255066, -1.3970065116882324, 0.5138066411018372, 1.62822687625885, -0.9410159587860107, -0.6849479675292969, -0.9235130548477173, -2.2725000381469727, -1.5234322547912598, -0.8505385518074036, 1.3625415563583374, 0.9501105546951294, 0.2544786036014557, 0.6297233700752258, -1.2173551321029663, -0.48428434133529663, 1.8473999500274658, 0.04240983724594116, 0.053355827927589417, 0.7220475077629089, 0.023244233801960945, 0.3186269998550415, -0.27939140796661377, -0.9295615553855896, 0.5620869994163513, -0.9716367125511169, -0.33976349234580994, 0.13997995853424072, -1.9949098825454712, 1.1102467775344849, 0.5860517621040344, 0.36966773867607117, 1.6178127527236938, -0.6026955842971802, -1.7769564390182495, -0.35287436842918396, 0.8392260074615479, -1.242177963256836, 0.9585480690002441, 1.3992844820022583, 0.7229321599006653, 0.5944830775260925, -0.6758373379707336, 0.9910869598388672, -0.11809921264648438, 0.8507984280586243, -1.3056832551956177, 0.47442004084587097, -0.5282788872718811, -1.0561312437057495, 0.8430632948875427, 1.4087378978729248, -0.1755644530057907, -0.1592426598072052, -0.5680689215660095, -0.923366129398346, 0.5626552104949951, -0.7801291346549988, 0.2024240344762802, -2.1387479305267334, -0.728900671005249, -0.25285089015960693, -0.8291441798210144, -0.6464948654174805, 0.6194171905517578, -0.5003846287727356, 2.660933017730713, -0.37418049573898315, 0.595858097076416, -0.1659945398569107, 0.06938507407903671, -1.7662684917449951, -1.5934938192367554, 0.09796440601348877, -0.622500479221344, 0.934961199760437, 1.9718643426895142, -1.5842087268829346, 0.24352698028087616, -0.8188858032226562, -1.122334599494934, -0.8848597407341003, -0.7619506120681763, 1.8028885126113892, -0.271023690700531, 0.019284741953015327, -0.659491240978241, 0.3235636353492737, -0.37689220905303955, -0.458402544260025, 0.9091106653213501, 0.14598606526851654, 0.21403710544109344, -0.18717345595359802, 1.0834498405456543, -0.30785486102104187, 0.05787765607237816, 1.282332420349121, -1.6671240329742432, 0.4499460756778717, 0.8579764366149902, -0.7990127801895142, -0.2988099157810211, -0.13123390078544617, -0.7597222924232483, 1.2053346633911133, 1.6606656312942505, 0.7332071661949158, 0.009777899831533432, -0.13773056864738464, 1.4881547689437866, 0.4194355905056, -0.972804844379425, 0.7367250919342041, -0.17202581465244293, -1.2127351760864258, -1.7017279863357544, -0.3156066834926605, 0.40961647033691406, -1.5657035112380981, 0.40517693758010864, 0.28051096200942993, 0.1687297523021698, 0.5785745978355408, 0.3121252954006195, 0.12270081043243408, 0.9202055931091309, 1.182904839515686, 1.0505130290985107, 0.1884182244539261, -0.7908434867858887, 1.9301503896713257, 1.425399899482727, -0.22202903032302856, -0.4278556704521179, -1.729292631149292, -0.3792726397514343, 1.1066629886627197, -2.0082027912139893, 2.671571731567383, 2.7871806621551514, 0.22512441873550415, -1.742230772972107, -0.18127892911434174, 2.574129819869995, -1.4334766864776611, -1.6801948547363281, 0.3532552421092987, -0.917053759098053, -2.6015830039978027, -0.7055251598358154, 0.058181557804346085, 0.09214900434017181, 0.22588826715946198, -0.30326375365257263, 1.8050060272216797, 0.2248251587152481, -0.34102970361709595, -0.32320886850357056, 0.9383430480957031, 2.1003382205963135, -1.7986598014831543, 1.2529231309890747, 0.6827921867370605, -0.9438403248786926, -0.5489206910133362, -1.1714526414871216, -0.5768239498138428, 1.6713517904281616, 0.4762972295284271, 1.2065237760543823, -1.3316504955291748, -0.2662380039691925, -0.11298533529043198, 1.7082222700119019, -0.14084888994693756, -0.022758014500141144, -0.37132585048675537, -0.6875933408737183, 1.4512699842453003, -0.20041342079639435, -0.5586815476417542, -0.42155638337135315, -1.119341492652893, -0.7643730640411377, -2.2135708332061768, -1.2606819868087769, -0.2533257007598877, -0.09547951072454453, 1.251243233680725, -0.3782990872859955, 0.705802857875824, 0.9586052298545837, -2.5797293186187744, 0.3752295672893524, 0.046308938413858414, -1.3852273225784302, -0.20769767463207245, -2.819286823272705, -1.8466644287109375, -1.3726098537445068, -0.8381109833717346, 0.34238141775131226, 0.2760867178440094, -0.16152489185333252, -0.4941346347332001, -0.7092846632003784, 1.8802733421325684, 1.1044700145721436, 0.027712011709809303, -2.0246853828430176, 0.23936598002910614, -0.33652734756469727, 1.1042890548706055, 0.0580449253320694, -0.9986838698387146, -1.135010004043579, -1.1836583614349365, -0.22202293574810028, 1.9633032083511353, -0.10749955475330353, 0.501083493232727, 1.559601902961731, -0.7786003351211548, -1.2987908124923706, 2.4693756103515625, 1.5854769945144653, 0.8301770091056824, 0.19009694457054138, 0.02463027648627758, -0.4350607097148895, 0.28709688782691956, 0.17987973988056183, -0.4677737355232239, 0.34487274289131165, -0.6050395369529724, 0.2974579930305481, 0.6048146486282349, -1.862288475036621, 0.48105522990226746, 0.1711582988500595, -1.7507123947143555, -1.1345679759979248, -0.21985232830047607, 0.9435245394706726, -0.4303143620491028, -1.902314305305481, 0.0826551541686058, -0.1873762011528015, 0.3433746099472046, -0.664090096950531, -1.1199672222137451, 0.6115724444389343, 0.6680024862289429, -1.022197961807251, 1.4007595777511597, 0.023264016956090927, -0.10650445520877838, -0.7370476722717285, 0.762295126914978, 1.1111174821853638, 0.6407070159912109, 1.5411490201950073, -0.1842830926179886, 0.10317949950695038, 0.6427465081214905, -0.46051064133644104, 1.816321611404419, -0.8370680212974548, 0.4163144528865814, -0.5513728260993958, 0.9552644491195679, 0.9232790470123291, -0.3543499708175659, -0.36991965770721436, -0.4320586919784546, 0.762286901473999, 0.0018537952564656734, -0.017553091049194336, -1.9292882680892944, 0.1365181803703308, -1.1188815832138062, -0.9438955187797546, -0.17829857766628265, -0.5855894088745117, 0.7139700055122375, -0.3251994550228119, 0.8502170443534851, -1.8892239332199097, 0.310557097196579, -0.4983366131782532, 0.8618038296699524, -0.8099870085716248, 1.087159276008606, 0.8156264424324036, -0.3155030310153961, 0.9554659724235535, 0.7572232484817505, 0.38328301906585693, 0.9342925548553467, -0.4318014979362488, 1.5660039186477661, 0.7733001112937927, 0.8559733629226685, -0.4293655455112457, -0.7333027720451355, -0.9108144640922546, -1.6793550252914429, 0.21806201338768005, 1.824851393699646, -1.9612797498703003, 0.5776848793029785, -0.7729842662811279, -0.6415752172470093], [1.732971429824829, -0.044480301439762115, -2.097903251647949, 1.8426225185394287, -0.018077222630381584, -2.2727413177490234, -0.3828294277191162, 0.25367555022239685, -0.2677249312400818, 0.8747483491897583, -0.49282413721084595, 1.081178069114685, 0.7288190722465515, -0.5172872543334961, -1.2294915914535522, 1.8925871849060059, 0.1005782037973404, 0.34457001090049744, 0.20923899114131927, -0.6276047825813293, -1.3312591314315796, 0.9494533538818359, -0.16359272599220276, -1.0331000089645386, 0.25307023525238037, 0.4288918375968933, -1.2257447242736816, -2.7458655834198, 0.4897797703742981, 0.8120332360267639, 1.4645642042160034, -1.2971351146697998, 0.1989830732345581, 1.4216148853302002, 0.4910108149051666, -1.5094752311706543, 0.344784677028656, 0.5324242115020752, 1.877710223197937, -0.5920733213424683, 0.29675278067588806, -1.4868443012237549, -1.7574644088745117, -0.8840383291244507, 0.7405534386634827, -0.252277672290802, 1.5133634805679321, 1.6788785457611084, -0.8721634745597839, 1.0513736009597778, 1.0315359830856323, 0.661643922328949, -0.8363787531852722, -0.512434184551239, 1.2694292068481445, -0.29148566722869873, -0.020591218024492264, -1.5014104843139648, 1.1701691150665283, -1.7739099264144897, 0.43267226219177246, 1.2528005838394165, -0.7245503067970276, 0.08185078948736191, -1.1492301225662231, 0.11389756947755814, 0.060086846351623535, 0.41322097182273865, -0.29313814640045166, -0.23521247506141663, 0.3944064676761627, -0.5300219058990479, -0.5725685358047485, -0.017498940229415894, 0.5517471432685852, -1.0912137031555176, -1.2283616065979004, 0.052242543548345566, -1.0342968702316284, -0.29780152440071106, 0.029539475217461586, -0.5556288957595825, -0.8689858913421631, 0.15335460007190704, -1.5767230987548828, -0.09724746644496918, -0.00782680232077837, 0.5034472346305847, -0.30293649435043335, 0.1486493945121765, -1.6420292854309082, -0.5543936491012573, 0.9617776274681091, -0.7001065611839294, 1.130308747291565, 0.2770443558692932, 1.3545502424240112, -1.1193805932998657, 0.7822323441505432, -0.8350943326950073, 1.5858770608901978, 0.023787157610058784, 0.8078020811080933, 0.6458643078804016, -0.2939489781856537, 2.2796218395233154, 2.0331308841705322, -0.03056027553975582, 1.251949429512024, -0.35031193494796753, -0.6501185894012451, 1.608285903930664, 0.5469468832015991, 1.5279028415679932, -0.461710661649704, -0.6094916462898254, 0.9097954034805298, -0.5507440567016602, -1.2763049602508545, 0.11607646942138672, -0.27909621596336365, 0.052006613463163376, 1.392525553703308, 1.5946403741836548, 0.6110398769378662, 0.31324613094329834, 1.4948279857635498, -0.41986119747161865, -0.5877835154533386, -0.2022962123155594, -0.8366363048553467, 0.6092211604118347, -1.395366907119751, 0.03455698490142822, -2.258416175842285, 0.05177503451704979, 0.12163807451725006, -0.23916852474212646, 0.1638910472393036, 1.3870939016342163, 0.20553353428840637, 0.3109979033470154, -0.08705887198448181, -1.0639089345932007, 0.4646625816822052, 0.22369268536567688, -1.4061901569366455, -0.9209679961204529, -0.054799698293209076, 0.12490740418434143, 0.9797537326812744, 0.33379483222961426, -0.2991531491279602, -0.04857408627867699, -1.3860671520233154, -2.2946617603302, -1.4173465967178345, -0.12069212645292282, 0.5257281064987183, -0.5595092177391052, -0.4396519958972931, -0.8483668565750122, -0.30002832412719727, 1.407299280166626, -1.0410289764404297, 1.4741320610046387, -1.608010172843933, 0.33743688464164734, 1.0454106330871582, 0.925237238407135, 0.7832997441291809, 0.41631558537483215, -0.8691468834877014, -0.9215652942657471, -0.14170104265213013, -0.7002612948417664, 0.056110065430402756, -0.570512056350708, 0.15698575973510742, 0.21536213159561157, 0.5419796705245972, -0.3550245463848114, 0.9576975107192993, -0.9037559628486633, 0.5376216769218445, -1.4240248203277588, -0.09375466406345367, 0.16058146953582764, 0.612515389919281, 1.8320016860961914, -1.0182862281799316, 1.0017260313034058, -0.478349506855011, -0.2030210793018341, -0.37194380164146423, 0.37325724959373474, -0.2157302051782608, 0.6371715664863586, 0.06406380981206894, 0.14045897126197815, 2.1866652965545654, -0.9457588195800781, 0.011118223890662193, -0.41546452045440674, 0.13342133164405823, -0.38434383273124695, -0.2307651787996292, -0.5281362533569336, -0.8952262997627258, -0.6272766590118408, 1.0141620635986328, -0.6808016896247864, 0.6424076557159424, 0.03751523047685623, -0.9335781931877136, -0.02733783796429634, -0.10051175206899643, -0.21320459246635437, 0.18712662160396576, -0.3239622414112091, 0.8696565628051758, 1.1593739986419678, 0.18433895707130432, -0.45171651244163513, 1.1439939737319946, 1.6360676288604736, -0.2462988942861557, -0.2505987882614136, 0.5219380855560303, -0.07936861366033554, 1.3548980951309204, -0.4429314434528351, -0.010276137851178646, 0.272835373878479, 2.6852147579193115, -0.5701495409011841, -0.6380366683006287, -2.220763921737671, 0.0821555033326149, -0.9670512676239014, 1.4189194440841675, 0.16945892572402954, 1.7416222095489502, -0.3044778108596802, 0.25175997614860535, -0.05456100031733513, -0.7717620134353638, -1.148141622543335, 1.1513309478759766, 0.9563683271408081, -1.6753817796707153, 1.64210844039917, 0.8026741147041321, 0.3488771319389343, -1.7698051929473877, 1.0481932163238525, 0.4098186790943146, 0.28113028407096863, -0.7113146781921387, 0.6489418745040894, 0.23666951060295105, 0.6366982460021973, -0.7947853803634644, -1.9447994232177734, 0.4373995363712311, -1.4431862831115723, 0.37557554244995117, 1.0143218040466309, 0.48296883702278137, 0.5257165431976318, 0.29265153408050537, 1.257555603981018, 1.4577436447143555, 0.2725374698638916, 0.8410516977310181, -0.28653088212013245, 0.07891425490379333, 1.3856388330459595, -0.9625517725944519, -0.5430514216423035, 0.4857286512851715, -0.5939691662788391, -0.17794033885002136, -0.7571221590042114, -2.2779288291931152, 0.15641003847122192, 2.3185794353485107, 0.062079206109046936, -1.5927374362945557, 0.612169623374939, 0.6474370360374451, -1.1212102174758911, 0.5233870148658752, -0.7114329934120178, 0.31730929017066956, 1.7776483297348022, -0.6337906122207642, 2.087191581726074, -2.3013288974761963, -0.1544926017522812, 0.6872209906578064, 0.6539828181266785, -1.1267117261886597, 0.4218778610229492, -0.6402594447135925, -0.21318593621253967, 0.7398490309715271, -0.39251309633255005, 0.1263384222984314, 1.9667006731033325, 0.7198560237884521, 1.3788045644760132, -1.3602322340011597, 0.6964221000671387, 1.5889617204666138, 0.6026039123535156, 0.011883889324963093, -0.6660791635513306, 0.3934418559074402, 1.166125774383545, -0.9776106476783752, 1.5850735902786255, 0.21599756181240082, -0.7588421106338501, -0.18974395096302032, -0.6023151278495789, 1.1438263654708862, -1.1180739402770996, 0.31484127044677734, 0.3620532751083374, -1.2163277864456177, 0.08280505985021591, -0.3418351113796234, -0.7960100769996643, -0.39201319217681885, -0.4669552445411682, -0.12669004499912262, -0.7661036252975464, 0.5109844207763672, -0.20336216688156128, 1.480088710784912, -0.22192242741584778, 2.06095290184021, -1.5400004386901855, -0.9635007977485657, 0.7733410596847534, 0.09072311222553253, -0.7486335635185242, -0.7266891002655029, 0.10433276742696762, -1.4950450658798218, 2.8427414894104004, -0.14082588255405426, -0.18557915091514587, 0.4967365264892578, 0.6258410215377808, -1.0531631708145142, -0.7703342437744141, 1.2817765474319458, 0.4778621792793274, 1.7348276376724243, 0.4108649492263794, -0.8569881319999695, 0.9826969504356384, 0.3063918948173523, -0.06815069913864136, -0.270005464553833, 0.8819891214370728, 0.19497661292552948, -1.5248124599456787, -0.10394856333732605, 1.50300133228302, -0.5327780842781067, -1.722878098487854, 0.2962377667427063, -1.3422400951385498, 0.28069907426834106, -1.168796181678772, 0.8781329393386841, 1.0948373079299927, 1.008154273033142, -0.6449012756347656, -0.38208258152008057, 0.5358680486679077, -0.35586342215538025, 0.5845622420310974, 1.1932545900344849, -1.7944483757019043, 0.6711702942848206, 0.5840550661087036, -0.48896902799606323, 1.5203980207443237, -0.6256542205810547, 0.008945178240537643, 0.19301451742649078, -0.7754673361778259, -0.7536032795906067, 0.5105814337730408, 0.0028106761164963245, -0.41942089796066284, 0.803938090801239, 0.5332109928131104, 0.17138685286045074, 1.5329211950302124, 0.264699786901474, 1.8940284252166748, -0.7766038179397583, 0.9875662922859192, -0.3268323540687561, -1.5480834245681763, 1.0630420446395874, 1.2637057304382324, -0.3235042095184326, -0.9871528148651123, -0.585967481136322, -0.24591198563575745, 0.09882219135761261, -0.012534253299236298, -0.6140228509902954, 0.6586722135543823, -0.6538248658180237, -1.4365330934524536, 0.0369439423084259, 0.42399904131889343, 0.7930657863616943, 1.4982426166534424, 0.7696918249130249, 1.275993824005127, 0.6653861403465271, 0.20744509994983673, -0.10019125789403915, -1.248486876487732, 0.4452633261680603, 0.6743952631950378, 0.03018376976251602, -0.007066064048558474, 0.6789069771766663, 0.8883177638053894, 0.12402557581663132, -0.2027747631072998, 0.5782791376113892, -0.914461076259613, -0.15940238535404205, -0.661855936050415, -0.7225638628005981, -1.369641661643982, -1.007285714149475, 0.2940104305744171, -0.2624404728412628, 1.5200761556625366, 0.8617709279060364, -1.3245253562927246, -0.9441755414009094, 0.04102838411927223, -0.5588483214378357, -0.268109530210495, -0.4958992600440979, -0.3119891881942749, 2.546238899230957, -1.366318702697754, -0.2687247395515442, 1.0340029001235962, -0.27279695868492126, 3.134507894515991, -0.34880495071411133, -0.4486548900604248, 0.2532814145088196, 0.03848031163215637, 0.9310587048530579, -0.9100696444511414, -1.0097253322601318, -1.0874465703964233, 2.01119327545166, -0.9448891282081604, -1.325555682182312, 1.0871446132659912, 0.3101300299167633, 2.7683303356170654, 0.6147387623786926, 0.09144038707017899, -0.23234066367149353, -1.2354928255081177, -1.7310973405838013, 0.6045781970024109, -0.16431733965873718, -0.15762096643447876, 0.4102133810520172, 0.31212326884269714, -0.6154322028160095, -0.5143324732780457, -0.40120187401771545, -0.980417788028717, 1.5836011171340942, 1.4136947393417358, -1.0844320058822632, -0.6062610745429993, 0.1258094757795334, -0.799026608467102, 0.7808372378349304, 1.5850571393966675, -0.008693026378750801, -0.6033958792686462, -0.5580956339836121, 1.5357556343078613, 0.8357029557228088, 0.12227664142847061, -1.0674175024032593, 0.550652801990509, 0.34878218173980713, 0.902442455291748, 2.0981881618499756, -0.47170183062553406, -0.2840269207954407, -0.7526403665542603, 0.6248509287834167, -1.1042590141296387, 0.8320032954216003, 0.3817380666732788, 0.050166599452495575, -0.33775150775909424, 1.0333508253097534, -1.253156304359436, -0.8434796929359436, 1.326993703842163, 1.7207744121551514, -0.2853095233440399, -0.04117722436785698, -0.15399707853794098, 1.0110324621200562, 0.3356661796569824, -1.80562162399292, -0.19841358065605164, -0.6453835964202881, -0.002222925191745162, -1.1377164125442505, 0.792762815952301, -1.1301445960998535, -0.614522397518158, 0.02133796364068985, 0.37467411160469055, 0.0430370569229126, -0.8506049513816833, 1.2683342695236206, 1.200609803199768, 1.7941861152648926, -1.338536262512207, 0.3432467579841614, 0.9274752140045166, 0.486480712890625, 1.945508360862732, 0.3116442561149597, 0.13966430723667145, 1.3261561393737793, -0.19420364499092102, 0.06993036717176437, 0.34841644763946533, -2.1611814498901367, 0.12803927063941956, -0.5971728563308716, 0.7940981984138489, -0.5324239134788513, -0.7356559038162231, -0.27024075388908386, 0.0036272460129112005, 0.17015966773033142, 0.3056221306324005, -1.4972141981124878, -0.9352384209632874, 0.7286710739135742, 2.0295772552490234, 1.0299361944198608, 0.3888745903968811, 0.16640421748161316, 1.1215097904205322, 1.2121362686157227, 0.867010235786438, -0.9140568971633911, -0.3714645802974701, -0.40425926446914673, 0.3592372238636017, -0.05685507133603096, 0.37092557549476624, 1.3223774433135986, -0.23016375303268433, -0.8273617625236511, 2.32106351852417, 0.8917720913887024, 2.4305717945098877, -0.031974684447050095, -1.3490654230117798, 0.4455558955669403, -0.372297465801239, -2.0826940536499023, -0.32146137952804565, -0.3522890508174896, 0.1745045781135559, 0.4889373779296875, -1.9482346773147583, 1.6432214975357056, -0.05512049049139023, -0.17159222066402435, -0.37632060050964355, -0.5074639916419983, 1.0773909091949463, -0.4004345238208771, 0.41352546215057373, -0.3077140748500824, -1.062317967414856, -1.4095749855041504, -0.6344371438026428, -1.5249803066253662, -0.33512741327285767, -0.41311976313591003, 0.9851707220077515, 1.6012057065963745, 1.352692723274231, -0.4621598422527313, -0.0418623648583889, 0.15998181700706482, 0.8852854371070862, -0.6431387662887573, 1.7872023582458496, 2.2615957260131836, 1.014416217803955, 1.5447551012039185, -1.2115609645843506, -0.5896143913269043, -1.4382041692733765, 2.3464338779449463, -0.1662464588880539, -0.022783903405070305, 0.9900590777397156, 0.7206788659095764, -0.7084394693374634, 2.4763858318328857, -0.8700128197669983, 0.23062366247177124, -1.9590930938720703, -0.037043601274490356, -0.2756839096546173, -0.21695004403591156, 1.4463157653808594, -0.3729884624481201, -1.1466827392578125, 0.47072169184684753, 1.272094488143921, -0.24568389356136322, -0.5392563343048096, -0.6145258545875549, 0.6117948293685913, 0.9547353982925415, 0.12075483053922653, -0.07533647119998932, 0.22803671658039093, -1.7583447694778442, -0.3420383334159851, -2.2905044555664062, -0.13035781681537628, 1.2835925817489624, 1.065159797668457, -1.3201712369918823, -0.7594497203826904, 0.671241283416748, -1.191283941268921, -0.029042400419712067, -1.1359217166900635, -2.2875380516052246, -1.6823652982711792, 0.602329671382904, -1.078552007675171, 1.1615577936172485, 0.6083864569664001, -0.5844563841819763, -0.648895800113678, -0.7046273350715637, -0.9671673774719238, 0.20719853043556213, -1.066928744316101, 1.4564675092697144, -0.7900313138961792, -1.0642192363739014, 1.723721981048584, 1.0527198314666748, -0.6363429427146912, 0.612475574016571, -1.2299760580062866, 1.1209825277328491, 0.6788012981414795, -1.546463131904602, -0.0371265672147274, 0.4494627118110657, 0.371685266494751, 0.5718737840652466, -0.22997520864009857, 1.0578540563583374, 0.6310126781463623, -0.1416141241788864, 1.9416049718856812, 0.3723219931125641, 0.7095510363578796, 1.1182888746261597, -0.007933052256703377, 1.0805861949920654, -0.5179159641265869, -0.38822683691978455, -0.1266045868396759, 0.47538426518440247, -0.5991504788398743, 0.04113344848155975, -0.5071905851364136, -0.8649688959121704, 2.048032760620117, -1.9588361978530884, -1.4113523960113525, 0.7704529762268066, -0.8497108221054077, 1.5985087156295776, -0.6957098245620728, 1.6520177125930786, 1.1507757902145386, 0.4999096095561981, -0.36002930998802185, -0.1154339462518692, 0.14774136245250702, -1.559633731842041, 1.030751347541809, -1.152322769165039, -2.2711853981018066, 2.159965753555298, -0.9129770994186401, -2.059332847595215, 0.03190920129418373, -1.0968163013458252, 0.4951848089694977, -2.765563726425171, -0.8604310154914856, -0.48134979605674744, 0.8583449125289917, -1.1940022706985474, -0.6066992282867432, 1.8088858127593994, 0.3499549925327301, -0.41580092906951904, 1.0485196113586426, -1.4728622436523438, 1.1303770542144775, -0.0999574065208435, 0.3816118836402893, 0.5424357056617737, 1.3365129232406616, -0.9956953525543213, 0.15648627281188965, 0.05669856816530228, 0.44735196232795715, 1.0842185020446777, 1.5608656406402588, 0.9452111721038818, -0.5662961006164551, -0.24837878346443176, 1.2845922708511353, 0.4123348891735077, 0.13214735686779022, -0.19590961933135986, -1.0097813606262207, 0.2725750505924225, 0.2343081384897232, -0.3612016439437866, -2.2343356609344482, -0.7303672432899475, -1.0512166023254395, -0.10262463241815567, 0.8547075986862183, 1.3416742086410522, 1.1786655187606812, -0.05256957933306694, -0.10659556090831757, -0.18995793163776398, -1.3926959037780762, 2.2504119873046875, 1.7846542596817017, -0.7692632079124451, 1.180359125137329, -0.6446868777275085, -1.3679207563400269, 0.5150675773620605, -1.0126570463180542, 0.5720531940460205, 1.1640870571136475, 0.22903482615947723, 0.3831251263618469, -0.290347695350647, -0.24067223072052002, 0.15362615883350372, -0.7536950707435608, 0.4671923518180847, -1.3491722345352173, 1.0575295686721802, -0.4680992364883423, -0.4796668291091919, -1.3924202919006348, 0.6188564896583557, 0.6898179054260254, 1.2770016193389893, 0.241341695189476, 0.1980394721031189, 0.7931540012359619, -0.510773241519928, -0.9865023493766785, 1.3613232374191284, -0.25093960762023926, -0.7937670350074768, 1.2736895084381104, -0.41859176754951477, -2.650169849395752, -1.4258856773376465, -1.0400835275650024, -0.4624585211277008, 1.043761134147644, -1.8372060060501099, 0.4001942574977875, -1.4636785984039307, -0.04547073692083359, -0.49511462450027466, 0.3539356291294098, 0.687761664390564, 0.030371317639946938, 1.3914810419082642, -0.16769817471504211, 0.009146908298134804, -0.3217306137084961, -0.41160908341407776, 0.213771253824234, -0.037720613181591034, 1.2570773363113403, -1.0041892528533936, 0.6229251623153687, -0.737962543964386, 1.0813416242599487, 0.9553830623626709, 0.05450829118490219, 0.028735311701893806, 0.4747805893421173, -0.3405129313468933, 1.804830551147461, -0.8580020666122437, -0.4691739082336426, -0.4188172221183777, 0.3325039744377136, -1.2295169830322266, -0.7207047939300537, 0.406310498714447, -0.13111457228660583, -0.4105421304702759, -1.0877494812011719, 0.7062280774116516, 0.6168503165245056, -0.8223831057548523, 2.3343331813812256, -0.25242266058921814, 0.4798077344894409, -1.6697468757629395, -0.9256135821342468, -0.6470430493354797, -0.048149775713682175, 2.829901933670044, -0.3206130564212799, -1.24979829788208, 0.3522176742553711, 0.4188629984855652, 1.4030109643936157, -1.902990698814392, -0.3600185811519623, 0.2064368724822998, -1.6815298795700073, -0.65143221616745, 0.7156795263290405, -0.29972338676452637, 0.5199793577194214, -0.49754199385643005, -1.921826958656311, 0.1477072834968567, 0.1345483958721161, 0.6350159645080566, -0.43542563915252686, 0.6675887703895569, 0.9988381266593933, 0.8111463785171509, 0.05091952905058861, -0.49239492416381836, -0.7088467478752136, 1.0034997463226318, 0.0883694514632225, 0.8031007051467896, -0.853335440158844, -1.6690077781677246, -0.27310049533843994, -0.6544689536094666, 0.25525084137916565, 0.5348432064056396, 0.737151026725769, -0.10269836336374283, -0.43049970269203186, -0.051305029541254044, -0.8219443559646606, 0.8727589249610901, -0.13108858466148376, -0.1302211582660675, -1.0192079544067383, -0.4380839765071869, 1.8044630289077759, 0.3037697374820709, 1.3044272661209106, 1.5153323411941528, -1.6489161252975464, 1.200019121170044, 0.49624374508857727, 1.010154366493225, 0.5873382687568665, -0.3475647270679474, 0.2061835676431656, -0.4833206236362457, 0.7143783569335938, 1.1333099603652954, -0.12664954364299774, -1.239699363708496, 0.020472709089517593, 0.4971209466457367, -0.8846102356910706, -0.6472973823547363, -2.044755458831787, 1.1533077955245972, -1.9874985218048096, 0.703557550907135, -0.38969555497169495, -0.34148213267326355, -1.5635285377502441, 0.3661777973175049, -1.203163743019104, 0.13243739306926727, -0.3001493811607361, 0.08000066131353378, 0.21507593989372253, -1.569382905960083, -0.7601369619369507, 0.19337697327136993, 0.6185855269432068, -0.33709201216697693, 1.599457859992981, 1.1348943710327148, -1.0745830535888672, 0.46327468752861023, 0.08437730371952057, 0.7616286873817444, 1.1869661808013916, 1.1367027759552002, -0.5719675421714783, -1.968775749206543, 0.9709179401397705, -0.32981595396995544, 0.1744185984134674, 0.7839798331260681, -0.890692949295044, 1.2934720516204834, 0.3987528085708618, -0.2998007535934448, 0.735791802406311, -0.12525604665279388, 0.3377658426761627, 1.060620665550232, -0.6342566609382629, -2.488718032836914, -0.6080999374389648, 0.2470899224281311, 0.8527572751045227, -1.880439043045044, 0.2073013186454773, 0.48055940866470337, -0.06421102583408356, 0.4141460061073303, 0.15740042924880981, -1.3651621341705322, -0.6034241914749146, 2.0537469387054443, 1.0599381923675537, 1.1975294351577759, -1.3365205526351929, 1.5619980096817017, 0.49041444063186646, 1.0048692226409912, 0.1808801293373108, -0.3818283677101135, -0.9857907295227051, 1.0783815383911133, 0.8967304825782776, 0.4509226977825165, 1.1651439666748047, 0.6144766807556152, 0.5426002144813538, 1.0093488693237305, -0.5341570377349854, -0.2975758910179138, 1.0095518827438354, -1.5417855978012085, -0.8862754702568054, -0.8765068054199219, -0.06815887242555618, -0.17632904648780823, -1.2395058870315552, -1.1881041526794434, -0.5706241726875305, -0.5430846810340881, 1.4848366975784302, -1.969893455505371, 1.329702377319336, 1.2522153854370117, 1.2544580698013306, 0.15722639858722687, -0.36488908529281616, 0.0010017304448410869, 0.672173798084259, 0.1204993799328804, -0.6442582011222839, -0.3304130434989929, -0.18729053437709808], [-0.7768265604972839, -0.23277586698532104, -0.8455320596694946, -1.0095638036727905, 0.8959588408470154, -1.404942274093628, -1.0116676092147827, -0.9025257229804993, -0.19616375863552094, 1.2391692399978638, 0.16553884744644165, 0.6935784220695496, 1.0725041627883911, 1.639129877090454, -0.3563404679298401, -0.33669331669807434, 0.291717529296875, 1.356157898902893, -0.026357606053352356, 0.3810461461544037, -1.1786969900131226, 1.3759087324142456, 0.05234288424253464, -0.06884381175041199, -0.3343927562236786, 0.8484264612197876, -1.2579762935638428, 0.8859583139419556, 0.6243484616279602, -1.445369005203247, -1.1262198686599731, -0.08923202008008957, -0.5689250230789185, -0.323955237865448, 1.102197527885437, -0.8486739993095398, -0.1344839185476303, -1.017703890800476, 0.6783942580223083, -0.8630214333534241, 0.7383003830909729, 0.6375898718833923, -0.793452799320221, 1.1184464693069458, 0.23766504228115082, -1.8500628471374512, 0.21415722370147705, -0.16467875242233276, 1.643649935722351, 0.7139747738838196, 0.29761695861816406, -1.6741163730621338, -0.5514877438545227, -0.5698059797286987, -0.037151746451854706, -0.7526926398277283, 1.4671802520751953, -0.11685692518949509, 0.5260013341903687, -0.19776137173175812, 1.6229223012924194, -1.0048918724060059, 1.3668761253356934, -0.9312310218811035, 0.8193147778511047, 0.6597399711608887, -1.1304775476455688, 0.6267276406288147, 0.6736186146736145, 1.1922833919525146, 1.1851519346237183, 0.4452272057533264, -0.7738462686538696, -0.33732157945632935, -0.4645519256591797, -1.0025279521942139, -0.19817884266376495, -1.9639990329742432, -0.1882869452238083, 1.7622531652450562, 0.6907145380973816, -0.2737182676792145, 0.37695759534835815, 0.10624232143163681, 0.783794641494751, -0.11230446398258209, 1.0057305097579956, 1.3273088932037354, 1.4456223249435425, 2.3512120246887207, -1.1908175945281982, 0.20655830204486847, 0.8084551692008972, -0.508150577545166, -1.1336803436279297, 0.6321420073509216, 0.3751547038555145, -0.8876342177391052, 0.4770461618900299, -0.4873730540275574, -0.14656007289886475, 1.816267728805542, -0.4079537093639374, -0.9155829548835754, -1.400189995765686, -0.1615312695503235, -0.25742799043655396, -0.39735883474349976, 0.7204710245132446, -0.7913569211959839, -0.10975850373506546, 1.5222036838531494, 0.4533191919326782, -0.17796537280082703, 0.38614651560783386, 0.264865517616272, -1.7186721563339233, -0.5567431449890137, -0.9157296419143677, 0.048251815140247345, 1.19109046459198, -0.12001945078372955, -0.7796827554702759, 0.8645898699760437, -0.04212165251374245, -0.43031540513038635, -1.0706084966659546, 0.4335322380065918, -0.5420734286308289, -1.0414366722106934, -0.034779295325279236, 2.1677939891815186, -0.0440266914665699, -3.174368143081665, 0.42110005021095276, -3.2182908058166504, -0.8076402544975281, 0.6982945203781128, -1.2322765588760376, 1.8731328248977661, 0.7534351348876953, -0.13067835569381714, 0.06170047074556351, 0.194067120552063, 0.24985189735889435, -1.699855089187622, -0.3265751004219055, -0.42674508690834045, -0.3677642345428467, 0.7295649647712708, -0.01742343045771122, 0.7592126727104187, -0.6905365586280823, -2.3036880493164062, 1.2648729085922241, -0.0009933954570442438, -0.032870300114154816, 1.2320092916488647, 0.8605309128761292, -0.5872490406036377, 1.0250517129898071, -0.3752589821815491, -1.4378209114074707, 0.06375004351139069, 1.689375400543213, 0.7265355587005615, 0.34231242537498474, -0.15582413971424103, -0.8403864502906799, -0.6631207466125488, 0.8473653793334961, 0.382455050945282, 0.4746040403842926, -0.3891419470310211, -0.6346868276596069, 0.8232619762420654, 1.5225213766098022, -0.04954305663704872, -1.9970402717590332, -0.9995968341827393, 0.4833492934703827, 1.1892446279525757, 1.0743199586868286, -2.0590624809265137, 0.8875537514686584, 0.7037931084632874, 0.8729903101921082, -0.9452579021453857, 1.7460768222808838, 1.7541261911392212, -0.7392148375511169, -0.060821812599897385, 0.30983415246009827, 0.02888467162847519, -0.8078882098197937, 0.005895291455090046, -0.5210058689117432, -0.8355867862701416, 0.5549901127815247, -0.04034984111785889, -1.3208050727844238, -0.36862432956695557, 1.1011080741882324, 0.3807283341884613, 1.2427780628204346, 0.6492904424667358, -0.7313942313194275, 0.28808262944221497, 1.575189232826233, 0.8032693266868591, 1.0648200511932373, -0.7158876657485962, 0.939522385597229, 1.073912262916565, -1.788200855255127, 1.3484878540039062, -0.40606942772865295, 0.07625861465930939, -0.29687896370887756, -0.46954071521759033, 0.40264010429382324, -0.8776964545249939, 0.2081659734249115, -0.7865850329399109, -0.32842007279396057, 2.0083401203155518, 0.6163791418075562, 0.2607395052909851, 1.4197895526885986, 0.544319748878479, -0.8717846274375916, -0.6995751857757568, 0.4634833335876465, -0.7199171781539917, 1.1581315994262695, -1.4238718748092651, -0.17275147140026093, -1.062294363975525, 0.7153062224388123, -0.7756390571594238, -0.4142208695411682, 1.5040085315704346, -0.9873760938644409, 0.04895684868097305, 1.3206926584243774, 2.009258985519409, -0.06266581267118454, -1.3653463125228882, -0.608193039894104, -0.5267013311386108, -1.2636359930038452, 0.9996508359909058, -0.7899706363677979, 0.3822375237941742, 0.16391445696353912, 1.8894760608673096, 0.21181920170783997, -0.6117832660675049, 0.07391593605279922, 0.4171869456768036, -1.721056580543518, -0.27191442251205444, 1.099722146987915, 0.23804859817028046, 1.6357499361038208, -0.16234402358531952, -1.152945876121521, -1.335841417312622, -0.14813938736915588, 1.1426122188568115, 0.32233601808547974, -1.0373188257217407, -0.6280829310417175, 0.6728605031967163, -0.26377496123313904, -1.312982201576233, -0.6210917234420776, -1.8937385082244873, -0.643632709980011, -0.7721200585365295, 0.7971157431602478, -0.19267459213733673, -1.699201226234436, -2.3797664642333984, 1.0679861307144165, 0.7008821964263916, -1.1982808113098145, -1.9881865978240967, 1.7450870275497437, 2.1311426162719727, 1.2123150825500488, 0.7074034214019775, -2.0634729862213135, -0.46801865100860596, 0.8441888093948364, -0.021847868338227272, -0.038487471640110016, -1.7388631105422974, -0.7600116729736328, 1.2297745943069458, 0.3126694858074188, 0.36418312788009644, -1.3552207946777344, 0.10205112397670746, 1.289786458015442, -0.6895914077758789, 0.5800930857658386, -0.4428609609603882, 0.4105819761753082, 0.10123232752084732, 0.282508909702301, 0.8206568360328674, 1.1924563646316528, 1.3455390930175781, 0.11798059940338135, -1.650559663772583, -0.1811908781528473, -0.2920706570148468, -0.1629343181848526, -0.11185000836849213, 0.16394799947738647, 0.04610753431916237, -0.14465080201625824, 0.5738109946250916, -1.1101524829864502, 1.560901165008545, 0.46544674038887024, -0.7280102968215942, 0.3502548038959503, -0.35492464900016785, 0.05733410269021988, 0.7945432066917419, 0.6251837015151978, -1.4594370126724243, -0.5688671469688416, -0.02280438505113125, -0.38397887349128723, -0.8425036668777466, -0.13991539180278778, 0.697149932384491, 0.8646959066390991, 0.24975275993347168, -0.2523238956928253, -1.476035237312317, 0.08826658874750137, 0.460347980260849, -0.534489095211029, 0.9526857137680054, -0.5211500525474548, -0.7701748013496399, -0.22834354639053345, 2.1289000511169434, -0.6350703835487366, -0.020621037110686302, -2.023880958557129, -0.10461407899856567, 0.08235319703817368, -0.11529046297073364, -0.07514560967683792, 0.666325569152832, 0.36033254861831665, 1.4798309803009033, 1.8982559442520142, 0.5513892769813538, -1.1335742473602295, 2.155256748199463, -1.3154864311218262, 1.369201421737671, -1.1505259275436401, -0.9768773913383484, -0.399699866771698, 0.1329195648431778, 0.16686174273490906, 0.850883424282074, -0.10039855539798737, -2.60788631439209, 1.5104484558105469, 0.9074105620384216, -1.505859613418579, 0.33075186610221863, 0.19628284871578217, -0.5066942572593689, 0.5428143739700317, 0.07603906095027924, 0.7954350709915161, -0.417682021856308, 0.6802124977111816, -0.8693737983703613, 0.12275943905115128, -0.25833460688591003, 1.3984801769256592, -0.347103476524353, 0.23721371591091156, 0.6834633350372314, -0.7639273405075073, -0.034235864877700806, -0.8825745582580566, 0.25768423080444336, -1.3642868995666504, 1.9651000499725342, 0.2954292297363281, -0.21618519723415375, 0.27376267313957214, 0.12354452908039093, -1.1419267654418945, -0.9514312148094177, 1.0038355588912964, 0.1583540439605713, -0.5919464826583862, -2.177344799041748, 0.10611963272094727, 0.028662702068686485, -2.3742082118988037, 0.7513140439987183, -1.279532790184021, 0.6564056277275085, 0.013052578084170818, 0.3014853000640869, -0.14702583849430084, -1.9757581949234009, 0.3661433160305023, 0.1832859367132187, -0.8713945746421814, -1.2109283208847046, 0.5099875330924988, -0.1108599603176117, -0.5249657034873962, -1.0104904174804688, -0.2092289924621582, 1.1092565059661865, -0.3518203794956207, -0.7068484425544739, -0.27524855732917786, -1.3724616765975952, 0.3464139699935913, 0.0004191604966763407, 0.17927299439907074, 0.32327547669410706, -0.4961507022380829, -0.9406275153160095, 0.7130067944526672, 0.04845723137259483, -1.0514124631881714, 1.1283817291259766, 0.14057248830795288, 0.7467789053916931, -0.7120357751846313, -0.732154905796051, 0.10416588187217712, -1.4027936458587646, -0.9089462757110596, 0.5348670482635498, 0.8489022254943848, 0.15163737535476685, -1.0340068340301514, -1.822816252708435, -0.4958394765853882, 0.27383551001548767, 1.1838760375976562, -0.9818622469902039, 0.5936660766601562, -0.9000902771949768, -0.10914346575737, -0.5785797238349915, -0.16858239471912384, -0.07292751222848892, -0.250181645154953, 0.8479357957839966, 1.526443600654602, -0.11462980508804321, -0.06617718189954758, -0.6221154928207397, -2.0617716312408447, -0.9709783792495728, -0.5205583572387695, -0.5303390622138977, 0.8490705490112305, 0.165199875831604, 0.9236727952957153, 0.24337701499462128, 0.35864847898483276, 1.3267464637756348, 1.8627171516418457, 0.053043778985738754, 0.7560160756111145, 0.9338701367378235, -0.029842320829629898, 0.7534950971603394, -0.4218009412288666, 1.107845664024353, -0.8324188590049744, -1.0888698101043701, 1.5639333724975586, 1.4603631496429443, 1.5021840333938599, 0.3236137330532074, 1.0483295917510986, 1.200636625289917, 1.366228461265564, 0.5984867215156555, -0.33977505564689636, -0.9085733890533447, -0.8049107193946838, -0.2373272329568863, -0.5617614984512329, -0.5792044997215271, 0.5732351541519165, -1.668129801750183, 0.2588231563568115, 0.41768503189086914, -0.4973703622817993, -0.4240671396255493, -1.722261905670166, -0.40350961685180664, 0.13604824244976044, -0.02987436205148697, 0.2000461369752884, -0.4569057822227478, 0.3015584647655487, -0.2818024754524231, -0.6004247665405273, 0.4828004837036133, 0.6086590886116028, 0.2358231544494629, 0.4167603552341461, 1.570296287536621, 0.3920840620994568, 0.5285570621490479, 0.3766663372516632, -0.9744479656219482, -0.43307408690452576, 0.06520784646272659, 0.48076730966567993, -0.9791936278343201, -0.07187264412641525, -0.08459892868995667, -0.5249365568161011, -0.9294827580451965, -0.7334641814231873, -0.6145150661468506, -0.36149027943611145, -0.7930129170417786, -0.9674607515335083, -0.827153205871582, 0.3793940842151642, 0.7827548980712891, 1.6677937507629395, 0.01464006770402193, 0.31139296293258667, 0.3577827215194702, 0.6151813864707947, -0.31503939628601074, 0.22550947964191437, 0.6760404706001282, -1.8799846172332764, -0.581047773361206, -0.027087384834885597, -0.534551739692688, 0.5348525047302246, -0.0885060653090477, 0.7775845527648926, -1.1231597661972046, 0.31475716829299927, 1.6556121110916138, 1.7023435831069946, -0.1380113810300827, -1.344228982925415, -1.5623962879180908, -0.24449190497398376, -0.7643435001373291, -1.5187512636184692, 0.927489161491394, 0.7801993489265442, -0.3598608672618866, 1.5292617082595825, 1.2151082754135132, -1.2814233303070068, -1.2194284200668335, 0.994543194770813, -1.753623127937317, 0.4906158745288849, -0.41110190749168396, -0.9242599010467529, -0.10760510712862015, -0.09531334042549133, -0.017857002094388008, 1.3245556354522705, 0.9271751046180725, -1.6910570859909058, 1.095823884010315, -0.4493292272090912, -1.383612036705017, 1.3980239629745483, -1.6778473854064941, -0.131659135222435, 1.833231806755066, 1.5332844257354736, 1.719640851020813, 1.150084137916565, -0.8065361976623535, 0.6675750017166138, 1.391211748123169, 1.6768512725830078, 0.1805972009897232, -0.3641088008880615, -0.43814265727996826, 0.7517760396003723, -0.31652599573135376, 1.2176464796066284, 0.17634136974811554, -0.03228725865483284, 2.268033027648926, -0.15161950886249542, -0.4447863698005676, 0.8673816323280334, 0.6950223445892334, 2.165555953979492, -1.2816472053527832, 1.020490050315857, -0.373148113489151, -1.3491387367248535, 1.5115058422088623, -0.7752273678779602, 0.8846407532691956, 0.7598394155502319, -0.6446871161460876, 1.0245260000228882, 0.7730770111083984, 1.8604708909988403, 0.0022621701937168837, -0.34255450963974, -0.6891835331916809, -2.0092551708221436, 1.3431915044784546, -1.6047821044921875, -1.5630697011947632, 1.0463894605636597, -1.8665454387664795, 2.164745569229126, 1.0898479223251343, -0.8449679613113403, 0.499693900346756, 0.7438068985939026, -0.9303176403045654, -0.009021579287946224, 1.663336157798767, 0.7891401648521423, -1.5031545162200928, 0.5202867388725281, 0.8244096636772156, 0.09057985991239548, 0.3672875165939331, -0.48025795817375183, 1.162217378616333, -0.7833284735679626, 0.8704453110694885, 1.697044849395752, -0.2692333161830902, -0.8061124086380005, 1.28313148021698, 0.33344826102256775, -1.1139354705810547, -0.0901588425040245, 0.4247867465019226, 0.7821494340896606, -0.8184753060340881, 0.6589340567588806, 1.4757062196731567, -0.07602103799581528, -2.4012198448181152, 1.5457335710525513, 1.7280888557434082, -1.0165141820907593, -0.8880338072776794, 0.4795185923576355, -1.8790404796600342, 2.2417824268341064, 0.8379658460617065, -1.799731731414795, -0.955618143081665, -0.12176285684108734, 0.6718544363975525, -1.3532718420028687, -0.8810950517654419, 0.4385775029659271, -0.3527069389820099, 0.2533245384693146, 0.9710748195648193, -0.10013539344072342, 0.7647382020950317, -0.1860789954662323, -0.15242142975330353, 0.17855705320835114, 1.565199851989746, 0.3803916573524475, -1.3057235479354858, 1.3422399759292603, -0.1882675737142563, -0.5344106554985046, -1.079141616821289, 0.4394204616546631, -0.3936619460582733, 1.6135482788085938, 0.41889020800590515, -0.44257864356040955, -0.11345542222261429, 0.9790676832199097, -0.005038065370172262, 0.5168075561523438, -0.17938771843910217, -2.922720432281494, 1.6785688400268555, 0.06427031755447388, -0.06787228584289551, -0.3714902102947235, 0.0912429541349411, -1.575304627418518, 1.612738847732544, -1.0453656911849976, -1.0774974822998047, -0.6061943173408508, -0.04289964959025383, -0.7756758332252502, -0.5508983135223389, 0.5512563586235046, -0.6179351210594177, -2.097712993621826, -0.04351583495736122, 0.5756106972694397, -0.6883760690689087, -0.2884838283061981, -0.13021981716156006, 0.8279796242713928, -0.39292943477630615, 0.9655855298042297, 0.5392870903015137, 0.375461220741272, 0.7371849417686462, 0.12367440015077591, -1.1114681959152222, 1.102147102355957, 1.0236730575561523, -0.2656067907810211, -1.85283625125885, -0.10732747614383698, 0.6375715136528015, -0.9566513299942017, 0.8583043813705444, -1.7217692136764526, -0.20419524610042572, 0.3985818028450012, 0.7316071391105652, 1.480443000793457, 0.24554529786109924, -0.8721293210983276, -0.25745144486427307, 0.132290780544281, 0.6063988208770752, 0.8336462378501892, -0.8686830997467041, -0.45904722809791565, -0.22005721926689148, 1.6216014623641968, -1.1918115615844727, 1.2304490804672241, 1.1206753253936768, -0.8358076810836792, -0.7307026982307434, -1.5157392024993896, -2.059520959854126, 1.3230000734329224, -0.39190757274627686, -1.2734113931655884, -0.8890379667282104, -0.9744028449058533, -0.15381498634815216, 1.0481648445129395, -1.2474850416183472, 0.7138896584510803, 0.731598973274231, -0.41520386934280396, 0.44200924038887024, -1.074713945388794, 0.8991200923919678, -1.422013282775879, 1.7093225717544556, 0.45405685901641846, -1.4166654348373413, 0.20001403987407684, 0.03319784626364708, 0.19602085649967194, 0.21248644590377808, -0.278321236371994, 0.44827955961227417, -0.4454706311225891, 0.7855473756790161, -0.12423982471227646, -0.8761419057846069, -1.0401611328125, 0.7029678821563721, 1.4767242670059204, 0.058968011289834976, -1.8231242895126343, 0.5438405871391296, 0.012705785222351551, 0.7243343591690063, -0.40962108969688416, 1.741369366645813, -2.044736623764038, -0.6917393207550049, -0.5804170370101929, -0.9012717008590698, -0.37235331535339355, -0.714227020740509, -0.2407563030719757, -0.17766518890857697, -1.608977198600769, -1.1268861293792725, -0.6586009860038757, 0.06870952993631363, -1.8735960721969604, 1.2567553520202637, 1.0571902990341187, 0.05501199886202812, -1.4056978225708008, 1.494640588760376, -0.19016103446483612, 0.6049015522003174, 1.31089448928833, -0.6399888396263123, -0.9162196516990662, -0.6490024924278259, 1.843709111213684, -1.3065128326416016, 1.2511178255081177, -0.0863746628165245, 1.482954740524292, -0.2613336443901062, 1.2129056453704834, 0.9984010457992554, -1.882655382156372, 0.529486358165741, 0.15823638439178467, -0.42473065853118896, 0.7595868706703186, -0.008726809173822403, -1.5329784154891968, 0.5120880603790283, 0.7056169509887695, 0.5751966834068298, -0.6964550018310547, -1.6679588556289673, -1.2779178619384766, 1.0446226596832275, -1.9533889293670654, 1.1603498458862305, -0.8619295954704285, 0.12688054144382477, 0.24330249428749084, 0.5818871855735779, -1.6003769636154175, -2.136688470840454, 0.2614474594593048, 0.4560425579547882, -0.9885267019271851, -0.3507223129272461, -0.7420691847801208, -0.3298003077507019, 1.196781039237976, -0.4967710077762604, 0.5700439810752869, -2.01485538482666, 1.4850759506225586, -0.50966477394104, -0.3869251608848572, 0.15245939791202545, -0.3139956295490265, 0.20320923626422882, -0.4268612563610077, -2.5642669200897217, -0.9068140983581543, 2.2798306941986084, 0.5994766354560852, 0.9292800426483154, -1.312454342842102, -0.5618041753768921, 0.6728309392929077, -1.8535207509994507, 1.5481630563735962, -0.23714523017406464, 0.38316917419433594, 1.3508464097976685, 0.5809016227722168, 0.4754161238670349, 1.5204393863677979, -0.3575519025325775, 1.477876901626587, -0.060419656336307526, -3.035961627960205, 2.089364767074585, 0.14447836577892303, 1.7527107000350952, 0.5871659517288208, -2.1345596313476562, -0.23503516614437103, 1.3008521795272827, 0.5281819105148315, 0.40355488657951355, 0.6169660091400146, 0.28840935230255127, 0.5156713724136353, -0.6004238724708557, -0.36232587695121765, 2.4056496620178223, -1.342781662940979, 0.7695039510726929, -0.7335008978843689, -0.7900790572166443, -1.515687346458435, 1.0327802896499634, 0.040354497730731964, 0.724928617477417, -0.7129701972007751, 1.669917345046997, -0.1428152322769165, -0.19655774533748627, 0.7358993291854858, 1.746944546699524, 1.1906332969665527, 1.889418125152588, 0.4806652367115021, 0.1561833918094635, 0.2562892436981201, -0.8795220851898193, 2.6322238445281982, 0.7534374594688416, -1.2120411396026611, -1.9552066326141357, -0.3345196545124054, 0.6140412092208862, -0.11328590661287308, -0.4540388882160187, 0.026231059804558754, -1.917382001876831, -0.3105575740337372, 1.248131513595581, -0.9424536824226379, 1.843692421913147, -0.0540241114795208, -0.7119156122207642, 0.005161442328244448, -0.7894806265830994, -0.5789036750793457, 0.710106372833252, 0.6219143271446228, 1.5135635137557983, 0.4258749485015869, -1.0055965185165405, -0.26497790217399597, 0.7105296850204468, 0.33545270562171936, -0.5060041546821594, 0.8850997090339661, -0.018993781879544258, 0.7020341753959656, 0.48381277918815613, -1.3661106824874878, -1.8520689010620117, 0.09595340490341187, 0.30649954080581665, 0.17890359461307526, -0.847278892993927, -1.3106794357299805, 0.1317698210477829, 0.9102336764335632, -0.7748918533325195, -1.4738528728485107, 0.30708038806915283, -0.7856993675231934, 1.3237926959991455, 1.3150321245193481, -1.1046733856201172, 1.1719011068344116, -0.30900025367736816, 0.24142086505889893, -0.7013009190559387, 1.8098434209823608, -1.2752387523651123, -1.0421310663223267, 0.5213727951049805, 0.7015864849090576, -0.24585191905498505, -0.6910093426704407, 0.3412477374076843, -0.8499022722244263, 0.19394274055957794, -0.48298415541648865, 0.3720744252204895, -0.034821394830942154, 0.6086432337760925, -1.949619174003601, 0.7318056225776672, 0.5683535933494568, -0.11939840763807297, 1.0114079713821411, -0.1559840589761734, -0.45112118124961853, -0.07541204988956451, 0.6401052474975586, -0.4667261838912964, -1.8034554719924927, -1.0677069425582886, 0.4487921893596649, 1.708440899848938, -0.6850221753120422, -0.25719931721687317, -0.18696241080760956, -0.3847256600856781, -1.6619136333465576, -0.10220212489366531, -0.7343739867210388, -1.3733501434326172, 1.7248907089233398, 1.693711757659912], [-0.7453756332397461, 1.1788724660873413, 0.2339750975370407, 1.2492554187774658, 0.21674694120883942, 0.14426776766777039, -0.08488711714744568, -0.35262665152549744, 1.3558361530303955, -0.4399798512458801, -0.2377701699733734, 0.6168479919433594, 0.6183677911758423, -1.4492326974868774, -1.1043953895568848, -0.5972222089767456, -1.6533688306808472, 0.395830899477005, -1.0655128955841064, 1.1950799226760864, 0.9606556296348572, 1.0578055381774902, 0.02154230698943138, -0.9676949381828308, -0.5721808671951294, -0.9515593647956848, -1.4580851793289185, 0.20196416974067688, 0.4474879503250122, 1.686192274093628, -1.3871996402740479, -0.6576512455940247, 0.239174023270607, -1.158897042274475, -0.8615437746047974, 0.7860099077224731, -1.161933183670044, -0.060012415051460266, 0.1822078973054886, 0.6571693420410156, 0.7604766488075256, 0.6110427975654602, 2.4143431186676025, -0.5033186674118042, 1.2721428871154785, -0.6233378052711487, 1.0313751697540283, 0.5195090174674988, -0.8778029680252075, 0.5670762062072754, -0.5798879861831665, -1.2805715799331665, -1.3082681894302368, 1.2684106826782227, 0.22691383957862854, 0.55258709192276, 0.4455769658088684, -0.5862596035003662, 1.8642058372497559, -0.7252708673477173, -0.5195549726486206, -0.5695194602012634, -1.4322788715362549, 0.476481169462204, 0.8240039348602295, -0.2955990135669708, -0.8645763993263245, -0.8928754925727844, 1.1954196691513062, -1.5398503541946411, 0.1359061598777771, -0.7899597883224487, 1.060426950454712, -1.142130970954895, 1.1327812671661377, 1.0850791931152344, 1.4534960985183716, -0.5304231643676758, 0.2522278130054474, 0.24276258051395416, 0.07925549894571304, 0.09363590180873871, -0.27797070145606995, 0.04949840530753136, -0.7945975065231323, 0.9664839506149292, -0.7635120749473572, 0.4665296971797943, -1.0342202186584473, 0.0858108326792717, 0.8849047422409058, 1.2606934309005737, -1.064120888710022, -1.417252540588379, -1.498406171798706, -0.22397151589393616, 0.9318420886993408, 1.6442147493362427, 0.9529626965522766, 0.196511909365654, -0.3703500032424927, 0.912518322467804, 2.010009765625, -0.49071982502937317, 1.4104011058807373, -0.3893927335739136, -1.564817190170288, -1.1707571744918823, -1.0542274713516235, 0.30885186791419983, -0.658686637878418, -1.5865486860275269, -0.7580406665802002, 0.5485129952430725, 0.5430822372436523, 0.10134284198284149, 0.31565049290657043, -0.04322303831577301, 0.40189802646636963, 0.3989242911338806, -0.210852712392807, 1.0051308870315552, 0.4756046533584595, 1.6740888357162476, 0.1866346001625061, -0.06445415318012238, -1.5422654151916504, 0.9097641706466675, 0.7805691361427307, -1.2608801126480103, -1.4993633031845093, -0.6723257303237915, -1.1266719102859497, 0.5718388557434082, 0.022650625556707382, -0.6796647310256958, -0.06392908841371536, 1.7602856159210205, -1.9082980155944824, -0.9015335440635681, -0.33273598551750183, -0.9090845584869385, -0.1948869228363037, -1.4891149997711182, 0.2861139178276062, -1.1859124898910522, -1.5370043516159058, 0.17349180579185486, 0.32803231477737427, -1.38139009475708, -1.0381056070327759, 0.4849880337715149, 1.1200973987579346, 0.5468155145645142, -0.9526870250701904, 0.6220167279243469, -0.49500253796577454, 0.23992623388767242, -0.16275283694267273, 0.8287066221237183, -0.8223693370819092, -0.2541426122188568, 0.6846275925636292, 0.9035216569900513, -0.98671954870224, -0.47422510385513306, -0.15655982494354248, -0.053850553929805756, -0.37492096424102783, -0.8527600765228271, 0.6351785659790039, -1.3162068128585815, 0.5734127759933472, 0.048350993543863297, 0.6580191254615784, 1.6009100675582886, -0.1491633802652359, 0.5971633195877075, 1.7141984701156616, 0.28022170066833496, -0.369102418422699, 1.102272868156433, -1.848955750465393, -1.24070405960083, -0.7325085997581482, 0.023336198180913925, -0.4793348014354706, 0.7716747522354126, -1.4774458408355713, -1.046684980392456, 1.3049843311309814, -0.760434091091156, 0.8445854783058167, -0.959915041923523, -1.4323596954345703, -0.2152782678604126, 0.9003353714942932, -0.4783380329608917, -0.27878695726394653, 0.20957735180854797, 0.6124168038368225, -0.17578089237213135, -1.4192012548446655, -0.06613732874393463, 0.8380564451217651, -0.1542498916387558, 0.7819737792015076, -0.11421161890029907, -1.0879344940185547, -0.2898206114768982, -1.1600654125213623, 1.6335067749023438, 2.4906909465789795, -0.5940459370613098, 1.1805088520050049, -0.871082603931427, 0.23914943635463715, -0.7126123309135437, 1.5926486253738403, 0.2638873755931854, 0.7848613858222961, 0.4059385061264038, 0.06937345862388611, -0.2993583381175995, 1.678396463394165, 1.0493870973587036, -0.7662800550460815, 0.9971243739128113, 0.8402004837989807, 0.577888011932373, 0.22921133041381836, -0.4160238206386566, -0.9344212412834167, -0.17087200284004211, 1.8600388765335083, 0.6322484612464905, -0.10418423265218735, -0.14829318225383759, 0.41220152378082275, -1.5149177312850952, -2.053717851638794, 1.7675495147705078, -0.7926900386810303, 0.44374504685401917, 0.978972852230072, 0.9516814351081848, 2.0236308574676514, -0.268034964799881, -0.48792922496795654, -0.5592210292816162, 0.7785226702690125, 1.0648430585861206, -0.09576617926359177, 0.1816474199295044, -1.336717128753662, -0.25071653723716736, 0.6064481735229492, -0.6661526560783386, -0.8049407601356506, -1.4582771062850952, -0.6892502307891846, 0.8804574608802795, 1.0423918962478638, 0.6957551836967468, -0.07919745147228241, -0.23183420300483704, 2.499671459197998, 0.8259103298187256, -0.9279048442840576, -0.6729264259338379, 2.502187490463257, -0.4970613420009613, -0.25133416056632996, -0.7313207983970642, -1.5308927297592163, -0.26608705520629883, 0.8097082376480103, -0.31781551241874695, 0.5571951270103455, 0.6323376893997192, -1.0007561445236206, 0.4286821186542511, 0.5527418255805969, -1.428017020225525, -1.100743293762207, -0.3682972490787506, -0.1345372200012207, 0.04887481778860092, 0.2659732699394226, 1.4659926891326904, 0.48974037170410156, 0.49340420961380005, 0.49938029050827026, 1.704936146736145, -0.13600847125053406, 0.3657040297985077, -0.3046731948852539, -0.2419431060552597, 0.3606142997741699, 0.7911055088043213, -0.3442392945289612, 0.9363690614700317, 1.8101136684417725, 1.508861780166626, 2.3648922443389893, -2.757460117340088, 0.23115339875221252, 0.9159696102142334, -1.9516181945800781, 2.8780596256256104, 1.178656816482544, -0.24690210819244385, -0.31637510657310486, -0.8604135513305664, -0.049424901604652405, -1.25490403175354, 1.0412665605545044, -0.23891478776931763, -0.37176427245140076, 0.7188672423362732, -0.3774910569190979, -3.622507095336914, 1.7349872589111328, 0.14241206645965576, 2.1888175010681152, -0.9301795959472656, 0.9513760209083557, -0.06492213159799576, 0.6911901831626892, 0.9140398502349854, 2.9720757007598877, 0.8698574900627136, 0.050462037324905396, -0.44225096702575684, -0.004489692393690348, -0.03550981730222702, 1.1922938823699951, 1.5170661211013794, -1.3278683423995972, -0.37137120962142944, 0.16848812997341156, 1.5922954082489014, -1.0557050704956055, -0.44042670726776123, 1.4871371984481812, 0.629231870174408, 0.2722766101360321, 0.36306455731391907, -0.9584936499595642, -0.9560239315032959, -0.6823045015335083, -0.28370073437690735, 0.15956531465053558, -0.31681394577026367, -0.09871789813041687, 0.04440172389149666, -1.7804044485092163, -1.1570829153060913, 0.3415277898311615, 0.6033748388290405, 0.31543222069740295, -0.6805431246757507, -0.5285211205482483, -1.244876742362976, 0.7304648160934448, -0.752671480178833, -0.9911370277404785, -0.38638219237327576, -0.3143082857131958, 0.4250950813293457, 1.115768551826477, 0.7340894341468811, -0.7439876794815063, -1.6725562810897827, 0.249113067984581, 0.4168444871902466, 0.6565497517585754, 0.23432062566280365, 0.48182013630867004, 1.0046278238296509, -0.041442885994911194, -0.8821301460266113, 0.05136062577366829, 2.0103650093078613, 1.6633676290512085, -0.3321169912815094, -0.5201199650764465, 0.2823408246040344, -0.1259908378124237, -2.005308151245117, -0.3071323037147522, -0.050975341349840164, 0.9538909792900085, 1.1247196197509766, -2.2305853366851807, 0.8584660291671753, -1.125555157661438, -0.05646970495581627, -1.2188936471939087, -1.0455399751663208, -1.591676115989685, -0.894494354724884, -0.37868377566337585, 0.4646734297275543, 1.0353435277938843, 1.421270489692688, 0.13204707205295563, 1.00856351852417, 0.2840551435947418, -0.8520671129226685, -0.5207898020744324, -0.4024936556816101, 0.06152358278632164, 0.3360311985015869, -1.7054762840270996, -0.11166884750127792, 1.0452603101730347, 0.10457969456911087, 0.29717499017715454, -1.4140520095825195, 1.5319650173187256, -0.6594734787940979, 1.2534151077270508, -0.3611985146999359, -0.23192648589611053, 0.9892608523368835, -0.1534283608198166, -2.4047203063964844, 0.4962139427661896, -0.3122532069683075, -0.4611847698688507, -0.9371303915977478, -0.2978414297103882, 1.2986069917678833, -0.4572995901107788, -0.19151361286640167, 0.6912580132484436, -1.0673917531967163, 0.5732223391532898, -0.09474052488803864, -1.7653347253799438, 0.301685631275177, 0.25409090518951416, -1.5379035472869873, -1.4225261211395264, -0.09803389757871628, -1.8207268714904785, -0.5054323673248291, 0.27819541096687317, 0.3694305717945099, 0.6104483008384705, -1.2171154022216797, 0.9737624526023865, 0.4557858109474182, 0.028120197355747223, -0.4829692840576172, 1.4613004922866821, -1.2882148027420044, -0.4287264049053192, 0.967822253704071, -1.4328577518463135, -0.7727262377738953, 0.5321419835090637, -0.6664639115333557, 0.6616628170013428, 0.852867066860199, -0.243448406457901, 1.3335224390029907, -0.6643267273902893, 0.7305582165718079, 1.947890043258667, -0.3305990993976593, -0.05086464807391167, -0.13488604128360748, -1.324458122253418, -1.0459868907928467, -0.007015241775661707, 0.44853657484054565, 1.5070116519927979, 0.4280809462070465, -0.991031289100647, 0.38271671533584595, -0.23210890591144562, 0.5617992877960205, -1.8761571645736694, 1.4646915197372437, -0.9474326372146606, -0.7146896123886108, -0.6321937441825867, -0.4435141682624817, 0.8189343810081482, -1.4633344411849976, -0.44321998953819275, -0.5572301745414734, -1.5479488372802734, -0.7987344264984131, -1.3451260328292847, 1.180276870727539, 0.8514335751533508, 0.5526329874992371, 1.1381487846374512, -0.7617456912994385, 1.2919589281082153, -0.6184291243553162, -1.8369060754776, 2.1299209594726562, 1.2280519008636475, 2.2891829013824463, 0.052515387535095215, 1.1378642320632935, -0.8164693713188171, -0.8755823373794556, 0.9209081530570984, 1.35434091091156, 1.0185441970825195, -1.4483096599578857, -0.46336129307746887, 0.597983717918396, 0.6428816318511963, -1.245378017425537, 0.14780375361442566, -0.2582646608352661, 0.4820469915866852, 1.3313292264938354, -0.8643164038658142, -0.3211486339569092, -0.5295202136039734, 0.03551050275564194, 2.187323570251465, -0.9555155038833618, 0.48349353671073914, 1.450100302696228, -1.0241382122039795, -0.053175706416368484, 1.701067566871643, 0.16792576014995575, 0.10267677158117294, -0.8611278533935547, -0.9201036691665649, -0.9223044514656067, 1.6202974319458008, -0.6397316455841064, 0.5737706422805786, -0.43623706698417664, 0.149558886885643, -1.3152433633804321, 0.6776137948036194, 0.9892675280570984, 1.6742175817489624, -1.970927119255066, -1.119272232055664, -0.5470134615898132, -1.9605646133422852, -0.46624261140823364, 0.7295620441436768, 0.5947511792182922, -0.14541466534137726, -0.49606600403785706, -0.8694193959236145, -1.1245752573013306, 0.8049150109291077, 1.0549826622009277, 1.2271013259887695, -0.8618743419647217, -0.28621944785118103, -0.6323158144950867, 1.6035559177398682, -0.7586970329284668, -0.09807668626308441, 1.450774908065796, -0.7206745147705078, -0.8342759013175964, -1.0346366167068481, 1.105639934539795, -0.54512619972229, 1.1507375240325928, -0.5400792956352234, -0.6613642573356628, 0.2715621590614319, 1.8942495584487915, -0.748468816280365, 0.47909680008888245, 0.4445365369319916, -1.1870211362838745, -0.27001386880874634, 1.0863475799560547, -1.578587293624878, 3.0265002250671387, 0.09482278674840927, -0.7996745109558105, -0.14847642183303833, 0.3763899803161621, 0.8921886682510376, 1.5001568794250488, 0.44841521978378296, -0.8244457840919495, -0.28095123171806335, -0.9968942403793335, -0.1869371235370636, 0.12121561169624329, -0.3092573285102844, -0.28700000047683716, -0.9246305227279663, 0.9447504281997681, 0.7847673892974854, 1.288785696029663, 0.14905603229999542, 0.37010976672172546, -1.3754072189331055, -1.5495433807373047, 0.6451072692871094, 0.08791649341583252, -0.7260306477546692, -1.2542641162872314, -0.4558893144130707, 1.3793853521347046, 0.30745643377304077, -1.1737699508666992, 0.5473630428314209, -0.4015602469444275, 1.286133885383606, 0.7886655330657959, -0.04408862814307213, -1.7280548810958862, 1.1102741956710815, -0.21618583798408508, 1.0657949447631836, 1.0903563499450684, -0.40502363443374634, 1.4362062215805054, -0.9198481440544128, -0.4221309423446655, 1.9958066940307617, -0.41529321670532227, 0.2749852240085602, 0.3786924183368683, -1.2080940008163452, -0.6737827658653259, 0.5741333961486816, 0.8481931686401367, 0.2595914900302887, -0.17953793704509735, 1.0161278247833252, -0.9278627634048462, -0.6329710483551025, 1.2790683507919312, 0.972368061542511, 0.9640824794769287, -1.5814733505249023, -1.5937658548355103, -1.5463532209396362, -1.2213544845581055, -1.1564147472381592, -0.012326635420322418, -0.051237404346466064, -1.7487529516220093, 0.8658265471458435, -0.9880626797676086, 0.9135939478874207, 0.2467886060476303, 1.19572913646698, -0.6208389401435852, 0.6096845865249634, 0.28557202219963074, 1.1250300407409668, -0.3809930384159088, 0.07640034705400467, -0.8160423636436462, -0.3928631842136383, 1.2109423875808716, -0.4862988293170929, 0.006311400327831507, 0.3163555860519409, 0.4234081208705902, -0.19562135636806488, 0.3384605944156647, 0.7359840869903564, -1.68808913230896, 1.3528610467910767, 1.2162526845932007, 0.7991250157356262, -0.13823844492435455, -1.4809786081314087, 1.1258426904678345, 0.024319427087903023, 0.9185603857040405, -0.6047435998916626, 1.438429594039917, -0.8195529580116272, -1.9916813373565674, -0.7159615159034729, -0.6020554304122925, -0.02388426475226879, -0.44460150599479675, -1.1461726427078247, -0.10405992716550827, 0.40166282653808594, 0.43377500772476196, 0.2799299359321594, -1.9506723880767822, -0.6608169078826904, 0.5578346252441406, -0.7989365458488464, -0.47795239090919495, 0.6402802467346191, -0.6476565003395081, -1.6252529621124268, 1.6677356958389282, -0.40356284379959106, -0.18593068420886993, -0.11412473022937775, 2.5870091915130615, -0.7923651337623596, -0.8173192739486694, 1.5334330797195435, 0.4104956090450287, -0.4959339499473572, -0.5735199451446533, -0.5129469633102417, 0.6873918771743774, 3.1027674674987793, -1.1241600513458252, 0.5437685251235962, -0.5768676400184631, 1.3930323123931885, -1.2181631326675415, 0.3661387860774994, 0.21512417495250702, 0.26813244819641113, -0.15391013026237488, -0.6193910837173462, -0.5851196646690369, 2.2480177879333496, -2.5939812660217285, 0.9572503566741943, -0.5748612880706787, 0.9185853600502014, 0.25296804308891296, -0.20790916681289673, -0.6869927048683167, 1.1577388048171997, 1.2570453882217407, -1.3964921236038208, 1.5078494548797607, -0.4749131202697754, 0.5811214447021484, -0.5515925884246826, -0.7846542596817017, -0.9339485168457031, -0.9312112927436829, -0.23616649210453033, -0.4985465407371521, -0.4177697002887726, 0.9542044997215271, 1.039422631263733, 1.1970895528793335, -0.3449822664260864, 0.8374810814857483, 0.5612772107124329, -0.9017685651779175, 1.4396406412124634, -0.6496166586875916, 0.9806562662124634, -0.9800425171852112, -0.7612835168838501, -0.9805997014045715, -0.5822816491127014, -0.931624174118042, -0.06639672070741653, 1.025499939918518, -1.9437373876571655, 1.069789171218872, 1.3728904724121094, -0.7011048793792725, 0.9493343830108643, 1.1902918815612793, 0.504866361618042, -0.0035603847354650497, -0.21495261788368225, 1.081454873085022, 1.665751338005066, -0.5329179763793945, -1.0326212644577026, -0.06338388472795486, -0.4869031608104706, 0.3333742320537567, 0.40688586235046387, -0.2867341637611389, 0.05495741590857506, -0.6395954489707947, -0.6003454327583313, -0.5179107785224915, 1.298339605331421, 1.1294373273849487, -1.070887804031372, 0.10861634463071823, 0.36194899678230286, -0.10902038961648941, -0.6130145788192749, 1.3865864276885986, 2.0999996662139893, 1.3198238611221313, -0.12538424134254456, 0.5345118045806885, 1.223121166229248, 0.4092887043952942, -0.34257379174232483, 0.3899051547050476, -0.5695911645889282, -0.5026527047157288, -1.3225072622299194, -0.46908849477767944, -0.6636179685592651, -0.5145663619041443, -1.0466175079345703, -0.8334619402885437, 0.8567712903022766, -1.1862528324127197, 0.4065348505973816, -1.202147126197815, 1.44520103931427, 0.4313211739063263, 0.22412778437137604, 1.1765241622924805, 0.04418171942234039, 0.13191933929920197, -0.01354445144534111, -0.6606742143630981, 0.4962915778160095, -0.4588167667388916, 0.03185047209262848, -0.8436284065246582, 0.2875787317752838, 0.696594774723053, 0.4453042447566986, 0.46997925639152527, -0.07840360701084137, -0.5158543586730957, 0.64463871717453, 0.20597602427005768, -0.38235729932785034, 0.08504680544137955, -0.02002323418855667, -0.3305250406265259, 0.01665240339934826, -0.3202221989631653, 0.04351744055747986, -0.25635263323783875, 0.303905189037323, -0.06908832490444183, 0.8679600358009338, -0.43803849816322327, 1.2409340143203735, -0.6959612965583801, -0.6705243587493896, 0.9930771589279175, 0.532155454158783, 0.6215430498123169, 0.45425018668174744, -0.79316246509552, 0.1919652819633484, -0.627243161201477, 0.22801245748996735, -0.2639918923377991, -1.1714006662368774, -0.2604937255382538, 0.2487877905368805, 0.7113211750984192, -0.05891488865017891, -0.07016569375991821, -0.5217424631118774, -1.0165181159973145, 0.8514397144317627, -2.019566774368286, -1.6520109176635742, 0.20816349983215332, 0.5058470964431763, -1.0415595769882202, -0.6960727572441101, -0.9110257625579834, 0.14726200699806213, -0.6430672407150269, -2.039801836013794, 0.779071033000946, 0.7815836668014526, -0.39286068081855774, 0.9791815876960754, -1.098903775215149, 1.8523907661437988, 1.0127882957458496, 0.4307028353214264, 1.484139084815979, 0.5229954719543457, 0.8242170810699463, 0.15911878645420074, 0.5841363668441772, 0.9001278281211853, -0.4225941300392151, 0.5761059522628784, -0.7504011392593384, -0.15571482479572296, -0.28222355246543884, -1.0419429540634155, 0.08416500687599182, -0.32544681429862976, -0.18894554674625397, -0.8942544460296631, 0.864808976650238, 0.9254857301712036, 1.330573320388794, 1.6487979888916016, 0.22207994759082794, -0.5628126263618469, -0.7697440385818481, 1.1585317850112915, -0.8100951313972473, -1.9112249612808228, 0.4810236394405365, 0.08861204236745834, 0.2811375856399536, -0.30014193058013916, 1.2674274444580078, 0.5583612322807312, -0.8236502408981323, 0.916982889175415, 1.272794485092163, -0.07301278412342072, -0.6111975908279419, 2.221013307571411, -0.9336963891983032, 0.6511231064796448, -0.09435375034809113, -1.7466163635253906, -0.3452840745449066, -2.3523850440979004, 0.19275270402431488, -1.3002489805221558, 0.9128166437149048, -1.091247320175171, -0.4584193527698517, 0.42911669611930847, 0.9749326109886169, 0.22385960817337036, 0.8096488118171692, 2.823408365249634, 0.29995808005332947, -0.07340045273303986, 1.076648473739624, -0.13759486377239227, 0.003927204292267561, 0.16708751022815704, -0.19503366947174072, 1.2071254253387451, -0.5383450984954834, -0.6119146347045898, 0.05811241269111633, -0.7356818914413452, 2.8002254962921143, 0.13118131458759308, -2.125462770462036, -1.738651990890503, -0.7765777707099915, -0.5129055380821228, 0.8931702971458435, -0.33047497272491455, -0.7889320850372314, 0.5270494222640991, -0.1797695904970169, 0.2832507789134979, 0.8391001224517822, 0.6009525060653687, 1.1648449897766113, -1.0444869995117188, -1.1246801614761353, 1.7022329568862915, -0.4115632176399231, -0.7195992469787598, 1.1211737394332886, -0.13535690307617188, -1.4346842765808105, -0.7234559059143066, -0.01733952946960926, -1.3824830055236816, -1.1037213802337646, 0.21309536695480347, 0.30281758308410645, 0.9707418084144592, -1.6586591005325317, -1.4245678186416626, 0.5945355892181396, 0.07999184727668762, -1.5838127136230469, -0.18100713193416595, 0.41333428025245667, 0.9080094695091248, -0.07262946665287018, -0.2738778591156006, 0.5955159664154053, 0.8881786465644836, 0.14601005613803864, 0.7451623678207397, -0.3514605760574341, 0.30397915840148926, -0.5993396043777466, 0.18213436007499695, 1.2413761615753174, 0.43769559264183044, 0.4221094250679016, 0.6445024609565735, -0.5186543464660645, -0.39066845178604126, 0.3962637186050415, -1.2158422470092773, -1.3221555948257446, 0.30764877796173096, 0.4031831920146942, 2.306699514389038, -0.35080960392951965, 0.4595533013343811, 0.6994971036911011, 1.0936827659606934, -1.1237688064575195, -1.1450165510177612, -0.49437013268470764], [0.4168853163719177, -0.7679243683815002, -0.2177446037530899, -1.1566301584243774, -0.026123981922864914, 2.5180771350860596, -0.36090922355651855, 1.8457057476043701, -0.46599894762039185, 0.9490760564804077, 0.37053897976875305, -0.9356517195701599, -0.4262945353984833, -0.02684270776808262, -0.4523864686489105, -0.21298977732658386, 0.20015135407447815, 0.9001720547676086, 0.8375241756439209, 0.4158972203731537, 0.9686599373817444, -0.7157840728759766, -0.35530683398246765, -0.8283262848854065, -0.04529929906129837, 0.7166866660118103, 0.676598846912384, 0.8030540347099304, 1.1102794408798218, -0.6552231907844543, -0.41946661472320557, 0.7557201385498047, 0.011132234707474709, 1.1872758865356445, -2.099011182785034, 0.06515026837587357, 0.5229772925376892, 0.7533770203590393, -0.07185531407594681, -0.08006597310304642, 0.45734313130378723, 0.1561524122953415, 2.3678081035614014, -1.750802755355835, -0.3764100670814514, 0.7117530703544617, -0.8073728084564209, 0.12785254418849945, -0.263025164604187, 0.05359077826142311, 1.0456922054290771, 0.5731560587882996, 0.08380696177482605, -0.41833579540252686, 0.405870258808136, 1.1494436264038086, 0.06568770110607147, -0.08562689274549484, -1.0829133987426758, -1.4289710521697998, 1.3786730766296387, 0.22206678986549377, -1.0941106081008911, 0.10785426944494247, -0.943364679813385, 1.185815453529358, 1.4909119606018066, -0.8430898785591125, 0.486855685710907, 0.9069179892539978, 0.024165872484445572, 0.8521322011947632, 0.5018889904022217, 2.1397180557250977, 0.30647221207618713, 0.8815428614616394, 0.05499032139778137, 0.7580556869506836, -0.37907275557518005, 1.4679434299468994, -0.8002772331237793, 1.1079723834991455, 0.5669875144958496, -0.5181393027305603, -0.7783039808273315, -0.29067978262901306, 1.8072445392608643, -2.2247495651245117, -2.188051462173462, -0.13409912586212158, 0.03376321494579315, 0.7697641849517822, 1.6673831939697266, 1.407249927520752, 0.4363861680030823, -0.16317176818847656, 2.3583972454071045, -0.4168052077293396, -0.546961784362793, -0.322174072265625, -0.38122236728668213, 0.29839882254600525, 0.2189406305551529, -0.6315426230430603, 0.45243069529533386, 0.37624603509902954, 0.1838410347700119, 1.750651240348816, 0.9677493572235107, 1.4638588428497314, -0.5604889988899231, 0.24913865327835083, 0.9954628944396973, -0.31225815415382385, -0.3974440395832062, -0.5309897065162659, 0.14366109669208527, -1.4454162120819092, 1.6403766870498657, 0.005305802449584007, 1.1491116285324097, 1.6134579181671143, -0.6792288422584534, -1.0217530727386475, -0.8538921475410461, -0.2655852735042572, 0.3360566794872284, -0.20750954747200012, -1.994189739227295, -1.5645358562469482, -1.783651351928711, 1.06514310836792, -0.7652247548103333, -0.5576138496398926, 0.7290806174278259, -0.978569507598877, 1.7543096542358398, -0.9802519083023071, 0.5648756623268127, -0.4712500274181366, 1.6697793006896973, 1.3239601850509644, -1.2532641887664795, -0.15899255871772766, -0.738064169883728, -0.8709336519241333, -2.147425651550293, -1.2745521068572998, 1.014393925666809, 0.8691620826721191, -0.9924790263175964, 0.06288982927799225, 1.4673409461975098, 0.5852094888687134, -0.3104109466075897, -0.9005275964736938, 0.9346622228622437, 1.7072402238845825, 0.6618858575820923, 0.5482085347175598, -1.491629719734192, 0.30629995465278625, 0.057390179485082626, 0.8186084628105164, -0.03787992522120476, -1.2162129878997803, 0.5755504369735718, -0.6436046957969666, -1.0197075605392456, 0.17165188491344452, -0.40169599652290344, -0.28696316480636597, -0.6458408236503601, -0.22249525785446167, -0.03221143037080765, 1.04964280128479, -1.1723746061325073, -0.49539950489997864, 0.06666487455368042, 0.267372727394104, -1.4359501600265503, 0.6593025922775269, -0.10365721583366394, -0.3630181550979614, 0.898552656173706, 1.5133229494094849, -0.19901128113269806, -1.739362120628357, 0.5045491456985474, -1.0918868780136108, -0.8793760538101196, 0.5650143027305603, -0.5559043884277344, -1.2204574346542358, -0.4217386245727539, 0.15758557617664337, 0.011010858230292797, -0.10153800249099731, 1.3784140348434448, -2.7691810131073, 0.6533522605895996, 1.0062458515167236, 0.8629952073097229, -0.8327222466468811, -0.48616793751716614, -0.5664423704147339, -0.12137366831302643, -0.41730695962905884, 1.3837354183197021, -0.041436582803726196, 0.7273499369621277, 1.4210405349731445, -0.6028181910514832, 0.7439252734184265, -0.13280005753040314, -2.519298553466797, -0.4290555417537689, 1.1249027252197266, 0.3086274266242981, -0.15057121217250824, -0.6706188321113586, -1.9697299003601074, -1.2196556329727173, -2.9308454990386963, 2.1467361450195312, -0.10762423276901245, 1.122969150543213, 0.9418107271194458, -0.0803627222776413, 0.9482656121253967, 0.27792832255363464, -0.3908436596393585, -0.3113958537578583, 2.0457403659820557, -0.5396761894226074, 0.2307562679052353, 0.5313618183135986, -0.1767578274011612, 0.8625338673591614, -0.47086790204048157, 0.4132455587387085, -0.3034276068210602, 0.7369099259376526, -0.47866290807724, -0.20049749314785004, -2.4323463439941406, 0.640693187713623, 0.18473370373249054, 0.16149993240833282, 1.1296696662902832, 0.6961945295333862, -0.32648393511772156, -0.9295742511749268, 1.1128082275390625, -1.5795314311981201, 2.2849743366241455, -0.4757045805454254, 0.26130563020706177, -0.15971961617469788, 0.39718765020370483, 0.20190021395683289, 2.3942742347717285, -2.8807992935180664, 0.634304404258728, 1.3703945875167847, -0.7053350210189819, -1.026306390762329, -0.2742317020893097, 0.4538304805755615, 0.4878048002719879, -0.547204315662384, 0.6573819518089294, -0.20713630318641663, -0.1269848793745041, -1.2924516201019287, -0.29520413279533386, -0.28314152359962463, 1.3309725522994995, -1.1316016912460327, -0.3691224157810211, -1.3086801767349243, 0.5127813816070557, 1.2394707202911377, 0.1685386449098587, -1.072736144065857, 0.09614748507738113, -0.3706258535385132, 0.8753968477249146, -0.13461297750473022, 0.4867550730705261, -0.15132500231266022, -0.04760100319981575, -0.589885413646698, 0.9411548972129822, 0.22542622685432434, 0.21848876774311066, -1.3620963096618652, 0.3060823082923889, -0.5396221280097961, -1.1018235683441162, -2.158982753753662, 0.6788637042045593, 1.7071613073349, -0.8822733759880066, -1.4837908744812012, -1.1153285503387451, -0.06287399679422379, 0.5885452032089233, 0.6061648726463318, -0.8876240253448486, -1.3046785593032837, -0.04393906891345978, 0.1895962804555893, -1.615494966506958, 0.6464753150939941, 1.2545630931854248, 1.4857889413833618, 0.9819362163543701, 0.2655920386314392, 1.985493779182434, -1.9330097436904907, 2.1038260459899902, 0.18146169185638428, -1.1039304733276367, 1.2187952995300293, 1.4224933385849, -1.4937634468078613, -1.034034252166748, 0.6193346977233887, -1.1512243747711182, -1.62450110912323, -0.33675339818000793, 0.14070510864257812, -0.18025322258472443, -0.8762177228927612, -0.7699875831604004, 0.2901429831981659, -0.6057506799697876, -2.993147373199463, -0.3787113428115845, 0.04776476323604584, -1.053242564201355, 0.9484354257583618, 1.2656729221343994, -0.9081760048866272, -0.27646416425704956, -0.39216798543930054, -1.6965935230255127, -0.5711890459060669, 0.5951135158538818, -1.306983470916748, -0.4073220491409302, -1.0686314105987549, 0.33005478978157043, 0.49266576766967773, 0.20431523025035858, -0.24465639889240265, -0.5230548977851868, 0.2459668666124344, 0.3064071834087372, -1.1376049518585205, 0.7445033192634583, -0.5115182995796204, -0.5005932450294495, 0.8973143100738525, 1.5270397663116455, -0.7809654474258423, -0.06872588396072388, -0.32168516516685486, -0.6706662774085999, 2.766474485397339, -1.445354700088501, 0.556058943271637, -0.37713828682899475, 0.712811291217804, -1.3563814163208008, -1.1098068952560425, 0.5839419364929199, 0.1730826050043106, 1.420128583908081, -0.08449253439903259, -0.6327988505363464, 0.3022019565105438, -0.8802830576896667, 0.06620591133832932, 1.8786579370498657, 1.6370958089828491, -0.25644436478614807, 0.07835103571414948, -0.5414238572120667, 1.5810763835906982, -1.447108507156372, -0.7166433334350586, 0.06728294491767883, -2.090926170349121, 0.9478258490562439, 1.0723049640655518, -0.41677355766296387, 0.07277318090200424, 0.7937890291213989, -0.4464227557182312, 0.39905664324760437, -0.8646724224090576, -0.9837391972541809, -0.4834625720977783, 0.6899757981300354, 0.11546169221401215, -0.052235253155231476, -0.5814175605773926, 0.625356912612915, -2.0740301609039307, -0.7343690395355225, 0.49611636996269226, -1.1694339513778687, -1.5930359363555908, -2.0275869369506836, 0.9847508072853088, 0.34603267908096313, 0.13821490108966827, 0.44063806533813477, 1.1385480165481567, -0.01481297705322504, -0.4237901270389557, -0.1872042864561081, -0.10130488127470016, 0.7432528138160706, 0.029515359550714493, -1.9902864694595337, 1.3197522163391113, 1.90004301071167, 0.686748743057251, 0.9962154626846313, -0.6646923422813416, 0.305652916431427, -0.6633038520812988, 1.9187915325164795, -1.381325125694275, 0.33408471941947937, 0.8129704594612122, 1.8417001962661743, 0.04674501717090607, 0.4349779188632965, 1.0184918642044067, -0.06601791828870773, -2.1838440895080566, -0.4240696430206299, -0.7238667011260986, -0.5452662110328674, -0.9205873012542725, 0.1205030009150505, 0.6321628093719482, 3.4078261852264404, 1.8983560800552368, -0.002876195590943098, 0.4421098530292511, -2.110682964324951, -0.1724950075149536, -1.0855318307876587, -0.30717143416404724, -0.7854227423667908, 1.4422212839126587, -0.2106131911277771, -1.3759616613388062, 1.225342035293579, 0.8348434567451477, -2.7646121978759766, -0.1314466893672943, -1.0425255298614502, -1.1719703674316406, -1.1279231309890747, -0.41218113899230957, 0.34435898065567017, 0.4241044819355011, -0.19452914595603943, -0.5037466883659363, -0.09895224124193192, 0.8172066807746887, -0.2194666862487793, 1.7397476434707642, -0.256733775138855, -0.09464555233716965, 0.3314303755760193, -1.7078880071640015, 0.27039623260498047, 1.1672714948654175, 0.2094745635986328, 0.042374346405267715, -1.2680602073669434, 0.5663114190101624, -0.10222572833299637, 2.014592170715332, 0.4044483006000519, -1.2276760339736938, 0.4372137486934662, 0.6702953577041626, -1.771391749382019, 0.9130078554153442, -0.27958059310913086, -1.8603709936141968, 0.210964173078537, -1.1712493896484375, 0.9666271209716797, 1.995723009109497, 0.4793449640274048, -0.7150890827178955, 1.2451910972595215, 0.010422764346003532, 1.1416929960250854, 1.2986031770706177, 0.3663184344768524, 0.017408760264515877, 0.2591743767261505, 0.2555965483188629, -1.254685878753662, 0.14163441956043243, -1.126009225845337, 1.4193015098571777, 0.4735487401485443, 2.1229918003082275, -0.4624668061733246, 0.22664982080459595, -0.7315179109573364, 1.4225785732269287, 0.0467001348733902, 0.010414550080895424, -0.43558499217033386, -0.5581192970275879, -0.11776016652584076, -0.6997700333595276, 0.12225831300020218, 0.7783296704292297, 1.8796794414520264, 0.9789987206459045, -0.11682097613811493, 0.7138288021087646, -2.0198466777801514, 0.20838218927383423, 0.7211752533912659, -0.5860928893089294, 0.7893581986427307, 1.8230220079421997, 1.0192781686782837, -0.9117453098297119, -1.1114747524261475, 0.3515097498893738, -1.8880482912063599, 0.4003432095050812, 0.3864237666130066, 0.8839271068572998, -0.07979096472263336, 0.7915380001068115, -0.8734990358352661, 0.8939211368560791, -1.6771661043167114, -0.8541202545166016, -0.7099093794822693, -1.6500728130340576, -0.09605904668569565, 0.5046249032020569, 0.9772949814796448, 1.7085280418395996, -1.363115668296814, -0.2272510826587677, 0.5352513194084167, -1.4037564992904663, -1.8329157829284668, 0.16845233738422394, 1.5115045309066772, 0.03446047380566597, -0.25478366017341614, -0.6088113784790039, 0.12767210602760315, -0.8226805329322815, -0.5229475498199463, -0.47433018684387207, -0.39745616912841797, 0.4542412757873535, -0.5466414093971252, 0.2765422761440277, 0.49107247591018677, 1.3249785900115967, -1.0786157846450806, -0.5645859837532043, -0.34240061044692993, -0.3944236636161804, 0.0895414799451828, -0.1765366643667221, 0.9264804124832153, -1.442121148109436, -0.3852338194847107, -0.7799848318099976, 1.785995364189148, 0.5012869238853455, 1.5749430656433105, -0.7856653332710266, -0.5615106821060181, -1.521380066871643, 1.3117051124572754, -1.983654499053955, 0.7402765154838562, -0.291830837726593, -0.6374067664146423, 0.5299225449562073, -0.8418071269989014, 0.6569578647613525, 0.22280211746692657, -0.1175651028752327, -1.7041698694229126, 2.5998148918151855, 0.5435341000556946, -0.9919596314430237, -1.271668553352356, 0.06343530863523483, -1.949005365371704, -0.40277886390686035, 0.23516970872879028, 2.438225507736206, 0.0038187343161553144, 1.7114678621292114, -2.805624485015869, 1.1549060344696045, 0.2887665331363678, -0.2909078598022461, 0.632016122341156, -0.4821162819862366, 0.7426463961601257, -0.42101043462753296, -0.3782225251197815, -1.5008326768875122, -0.08829757571220398, -0.7785203456878662, 1.9839648008346558, -1.9000052213668823, -0.34316983819007874, -0.5075374841690063, 0.4753464162349701, 1.2127259969711304, 1.4487274885177612, -0.6971781849861145, -0.7318202257156372, -2.041365146636963, -1.889938235282898, 1.891477108001709, 0.9616411328315735, 0.705091655254364, -1.8830780982971191, -0.47060784697532654, 0.8886591792106628, -0.3795951008796692, -0.8258056640625, 0.41747865080833435, 0.4632129967212677, 0.21185888350009918, 0.2814852297306061, 0.06430894136428833, -0.18380160629749298, -0.1497921198606491, -0.08775241672992706, 2.1244163513183594, -1.39149010181427, -0.5577006340026855, -0.3693707585334778, -0.1695539802312851, 1.5455995798110962, -0.3389355540275574, -0.552991509437561, 0.11689342558383942, 0.3512883484363556, 0.3662836253643036, -1.0764130353927612, -0.7692614793777466, -1.8417301177978516, -1.3466269969940186, -0.28089097142219543, -0.8042850494384766, 0.6923884153366089, 1.859497308731079, -1.1610394716262817, -1.875807762145996, -1.1757135391235352, 0.3806604743003845, 2.4523956775665283, -0.3296455144882202, 0.4783831834793091, -0.2289622724056244, -0.25409194827079773, 1.9107838869094849, -0.19057898223400116, -0.9266594052314758, 1.032072901725769, 1.55080246925354, -0.43639659881591797, 1.648903250694275, 0.3935147225856781, 0.6513515710830688, 0.4126104414463043, 0.37751659750938416, 0.0330357551574707, 0.18307507038116455, 0.7305172681808472, -1.2445311546325684, 1.0366628170013428, 0.8560444116592407, 0.5850359797477722, 0.6697070598602295, 0.766813337802887, 0.46643197536468506, -0.8937488198280334, -0.3257196247577667, -1.2781378030776978, 0.5590831637382507, -0.2896433174610138, 0.5441874861717224, 0.2862633764743805, -0.8427160978317261, -0.6372067332267761, -0.36196234822273254, 0.06409204751253128, -2.5820252895355225, -0.32944750785827637, -0.415876179933548, -0.1540900617837906, 1.715090036392212, 0.03926939517259598, -1.5301803350448608, -2.336923837661743, -1.7647168636322021, 0.3213767409324646, 0.48429593443870544, 1.1185076236724854, 0.8382466435432434, -1.3346718549728394, -0.3194817006587982, -0.5942682027816772, -1.904391884803772, 1.0904628038406372, 0.307125061750412, 0.4004794955253601, -0.13411609828472137, 0.43229940533638, -1.1584261655807495, -1.0569380521774292, -2.033392906188965, 1.1648908853530884, -0.19186559319496155, -1.546579360961914, 1.6235487461090088, 0.6783381700515747, -0.4481724798679352, 0.7748623490333557, -0.4351746439933777, -1.450965166091919, -0.6480786800384521, 0.6152316927909851, 0.706637442111969, 1.3550410270690918, -0.9532865881919861, 0.7516114711761475, 0.671755850315094, -2.0812454223632812, -0.19529232382774353, 1.0886399745941162, 0.34210333228111267, 0.378816694021225, 0.7185434103012085, -1.1344077587127686, 0.07725172489881516, 0.0923510417342186, -0.16089728474617004, 0.043515630066394806, 0.9890438318252563, 0.34724774956703186, -0.49332550168037415, -1.5103132724761963, -2.304544687271118, 0.7323504686355591, -1.0172314643859863, -0.35835278034210205, -0.2816326320171356, 1.5699620246887207, -0.3118128478527069, 0.8089163899421692, -1.8742492198944092, 0.5858110785484314, -0.3546934127807617, 1.169594645500183, -0.670562207698822, 0.7433302402496338, -0.8608207702636719, 0.194893017411232, 0.020758923143148422, 0.7411079406738281, 0.4235805869102478, -0.4269009828567505, -0.7862861752510071, -0.6847583651542664, -0.20849215984344482, 0.36731651425361633, -0.17232468724250793, 0.05628985911607742, 1.7641879320144653, -0.3206269145011902, 0.6697062849998474, -0.33623597025871277, 0.4697280824184418, 0.6630138754844666, 1.4705331325531006, -0.20403732359409332, -1.8516892194747925, 0.42631658911705017, 0.32975488901138306, -1.6100660562515259, -1.1387617588043213, -0.6016075611114502, -1.3347595930099487, -0.5450965166091919, -0.2873992323875427, 2.099153757095337, -0.20442675054073334, 2.4082233905792236, -0.3128853738307953, 0.08720103651285172, 0.12770609557628632, 0.18772795796394348, 1.7110071182250977, 0.890300989151001, -0.9803771376609802, -1.2714345455169678, 0.7351856827735901, -0.16012054681777954, 0.6930190324783325, 0.8707172274589539, -0.5801178812980652, 0.7171788215637207, -1.4824573993682861, -1.4101848602294922, -0.5597060322761536, -1.2767947912216187, -1.1200535297393799, -0.029801856726408005, -1.0553984642028809, 0.5077498555183411, 1.0103071928024292, -0.6758021116256714, -0.610207200050354, 1.3538472652435303, -1.4345693588256836, -0.6496497392654419, -1.0486422777175903, -0.1340533047914505, 0.49151647090911865, 0.18156713247299194, -1.0942260026931763, 0.10451358556747437, -1.1930930614471436, 0.2744186520576477, -1.1939014196395874, -0.5042303204536438, -0.010229574516415596, 0.850537121295929, 0.5787045955657959, 0.42808791995048523, -1.5391207933425903, -0.016093790531158447, 0.17478452622890472, -0.12227677553892136, 0.7807439565658569, -0.6872969269752502, 0.005519229918718338, 0.5639392137527466, 0.540795624256134, 1.4834920167922974, -0.908122181892395, 0.9657158851623535, -0.062108058482408524, -0.1620863676071167, -1.6800169944763184, -0.21005329489707947, 0.6187112927436829, -0.7266923189163208, 0.0164660457521677, -0.8686084151268005, 0.6080067753791809, -0.5067548155784607, -1.4172981977462769, 1.4076557159423828, -0.8493197560310364, 0.027130266651511192, -1.272637963294983, -0.8747580647468567, -1.2558223009109497, -0.8733006119728088, 0.9917685985565186, -2.324711322784424, 0.6883199214935303, -0.5027619004249573, 0.17653681337833405, 1.2891845703125, 0.17020027339458466, 1.1430861949920654, -0.753670334815979, 1.1085093021392822, -0.8633063435554504, -0.21445530652999878, 0.5033424496650696, -0.05040843412280083, -1.0254762172698975, 1.1501107215881348, -0.20767338573932648, 1.1991685628890991, -0.39654621481895447, 2.2080307006835938, 0.04378847777843475, -1.1373072862625122, 2.0744411945343018, 0.5629599690437317, 0.5102023482322693, -0.5208328366279602, -0.7337232828140259, -0.0035662853624671698, 0.15416854619979858, -0.4679136872291565, -0.24805092811584473, 1.1936345100402832, 0.0896638035774231, 0.18104170262813568, -0.008665340021252632, 1.8371349573135376, -0.22230885922908783, 1.4930115938186646, 0.1624746322631836, -1.2855863571166992, -1.7056750059127808, 1.8912968635559082, 1.62981116771698, -0.12804768979549408, 0.23582500219345093, -0.16518351435661316, -0.8662748336791992, 1.1388181447982788, 0.2782226502895355, -0.5506177544593811, 0.8932248950004578, 1.7498345375061035, 0.2563079595565796, 0.5180698037147522, -0.15823930501937866, -0.009669054299592972, -1.3198920488357544, -0.8370831608772278, 1.444730520248413, 1.5910859107971191, 1.187906265258789, 0.4803004264831543, -1.5914740562438965, -0.1421414613723755, -0.050447311252355576, 1.0027146339416504, -0.6042895913124084, 1.592234492301941, 0.8669710159301758, 0.9178049564361572, 0.6377310752868652, 0.01029419619590044, 0.1579921692609787, -1.0279407501220703, 0.033478397876024246, -1.7352668046951294, -2.2835514545440674, -0.33023396134376526, -0.18543314933776855, -0.4008674919605255, -0.567157506942749, -0.07904260605573654, 0.17924128472805023, -1.2966402769088745, -0.6055606007575989, 1.078674077987671, -1.0497373342514038, -0.4267728924751282, 0.8298500180244446, -0.21458956599235535, -1.6111609935760498, 1.3446826934814453, -0.8887829184532166, -0.317648321390152, -0.33063262701034546, -0.42364200949668884, -1.2275437116622925, -2.212819814682007, -1.586917757987976, 0.3793363869190216, -2.388997793197632, 0.605421245098114, -0.9635354280471802, -0.22847618162631989, -0.4918568432331085, 0.1515553593635559, -0.6218022704124451, 0.013634025119245052, -0.40028616786003113, -0.33896535634994507, 0.7869610786437988, -1.0594100952148438, 0.62483149766922, 0.4820977747440338, -0.30687204003334045, -0.2974960207939148, -0.5612791180610657, 0.19207431375980377, -0.1222875639796257, 0.38981959223747253, 0.9108704328536987, 1.6182345151901245, 0.01566772349178791, -0.5362358689308167, -1.5146994590759277, 0.23301266133785248, 0.8058410882949829, 0.8580443263053894, -0.5405330657958984, 0.06835271418094635, 0.47708600759506226, -0.02936534583568573], [-0.3244946300983429, -2.392767906188965, 1.0671998262405396, -0.14258421957492828, 0.3249753713607788, -0.9110503196716309, -0.40972331166267395, 2.087740182876587, -0.935085654258728, -2.3909597396850586, -0.6144404411315918, 0.7189351320266724, 0.6642684936523438, -0.1811453253030777, 0.8107455372810364, 1.0293227434158325, 0.13837292790412903, 1.2722834348678589, -0.1950860470533371, -1.1409785747528076, 1.163493275642395, -0.14789161086082458, 0.30312851071357727, -1.3414192199707031, -0.5178822875022888, 0.0705861747264862, 1.4651366472244263, -0.41783130168914795, 1.3352597951889038, -0.029900582507252693, 1.9046671390533447, 1.414019227027893, 1.958586573600769, -0.4234185218811035, 0.9697379469871521, -0.5066261887550354, -0.808770477771759, -0.38905075192451477, -0.3376743793487549, 1.924752950668335, 1.7945550680160522, -0.14145776629447937, 0.1570841670036316, 1.4607666730880737, 0.544770359992981, -0.3430198132991791, -1.7893213033676147, -0.2911489009857178, -0.6557983756065369, -0.7139965891838074, -1.0034915208816528, 0.2884737551212311, -1.3514436483383179, 0.6145645976066589, -0.230930358171463, 0.3552416265010834, 0.5408722758293152, -0.9486810564994812, -0.7552900910377502, -0.42930880188941956, 0.5079705715179443, 2.5493857860565186, -1.6960275173187256, -0.5260897874832153, 0.4809727072715759, -0.5803472399711609, 0.594367504119873, 2.4334359169006348, 0.2902264893054962, -0.09674864262342453, -0.1904439926147461, 1.377252221107483, 1.0703619718551636, 0.0015426514437422156, -1.0923210382461548, 0.30495011806488037, 0.05279461666941643, -0.09777437150478363, -0.948970377445221, 0.5442700386047363, -0.24681530892848969, -0.07251648604869843, -1.7392128705978394, 0.6414986252784729, -0.5352010130882263, 0.3665519952774048, 2.164290428161621, 0.10035694390535355, 0.22597110271453857, -0.20259693264961243, 0.5631236433982849, -1.1074496507644653, -0.9020426869392395, 0.5509436726570129, 0.5479588508605957, 0.5764079689979553, 1.7279402017593384, -0.5932639241218567, 0.7934167981147766, -1.4676120281219482, 0.30958566069602966, -0.5990854501724243, 0.3068675100803375, -1.2001434564590454, -0.9090385437011719, 1.0668864250183105, 0.009897778742015362, -0.07471892237663269, -0.6354221105575562, 1.6872854232788086, 1.8341985940933228, 2.1238365173339844, -1.502416729927063, -1.9102692604064941, -0.5202864408493042, -0.8935660719871521, -0.6466132402420044, -0.07366136461496353, -0.12091183662414551, 0.12205103039741516, 0.9212886095046997, -2.1993637084960938, -0.6117919087409973, 1.367796778678894, -1.5526533126831055, -0.04894978553056717, -0.9363150596618652, -0.9541907906532288, 1.3258463144302368, 1.0030560493469238, -0.06076009199023247, 0.49979954957962036, 0.28643015027046204, -0.9021552205085754, 1.0733833312988281, 1.1385363340377808, 1.589860439300537, 0.5963372588157654, -0.8495152592658997, 1.1248855590820312, -0.7211430072784424, -0.8484959602355957, -0.27512186765670776, -2.011814832687378, -0.04274776950478554, 0.4460214078426361, 0.27882689237594604, 0.8007017374038696, -0.21353618800640106, -0.16319122910499573, 0.0013249991461634636, 0.2159426063299179, -0.7336497902870178, 0.9037141799926758, 1.5637692213058472, 0.7698810696601868, -0.09782768040895462, 0.9034453630447388, 0.3778703212738037, -0.36952608823776245, 0.16549110412597656, 0.37555450201034546, 0.05404113605618477, -0.9963348507881165, 1.2968018054962158, -0.19760118424892426, -0.5856653451919556, 0.7371335029602051, -0.9074652194976807, 0.7916528582572937, -0.6950575113296509, -0.7369754314422607, 1.2144205570220947, 0.5965113639831543, 0.05859629809856415, -0.15275631844997406, -1.0649949312210083, 1.036286473274231, 0.2928754985332489, -0.2198101431131363, 0.797053337097168, 0.9599385857582092, 0.4565466642379761, 0.32503044605255127, 0.08719738572835922, 1.2015734910964966, -0.8086804747581482, 2.2475502490997314, 0.06986736506223679, 0.07249654084444046, -0.21415495872497559, 1.2985639572143555, 0.9870485067367554, -1.1378580331802368, 0.5053854584693909, -0.5796724557876587, 0.692139208316803, 0.5056810975074768, 1.375440239906311, 1.8893588781356812, 1.1481833457946777, -0.3270905315876007, 1.1825841665267944, -0.9561851024627686, -0.21570256352424622, -0.7343049645423889, -0.1930873692035675, 1.3127555847167969, -0.008633757010102272, -0.27988702058792114, 0.756861686706543, 0.537183403968811, 1.4384984970092773, -0.5794394016265869, -1.9310847520828247, -0.8239266872406006, 1.0778664350509644, -0.28021547198295593, 0.48206737637519836, -0.2443380504846573, 0.5240825414657593, -0.40178462862968445, -1.0250662565231323, 1.4968533515930176, 0.9751980304718018, -0.9382973909378052, -0.6214787364006042, 0.39450880885124207, 2.596292734146118, -1.5136258602142334, -0.7267800569534302, 0.526326060295105, 0.7991887331008911, 1.9225144386291504, 1.6875178813934326, 0.6077589988708496, -0.06338120996952057, 0.7911738753318787, 1.0718697309494019, 0.35399454832077026, -0.9978864789009094, -0.14014177024364471, -0.25468599796295166, -0.3917423188686371, -1.0160356760025024, -0.9433397054672241, -1.1687904596328735, -0.495487779378891, 1.5898798704147339, 0.1408066749572754, -0.5462494492530823, 2.0951693058013916, -0.17245592176914215, 0.44340869784355164, 0.37040427327156067, 1.85740065574646, 0.22263815999031067, -0.7635505795478821, 0.7354680299758911, 0.5705294609069824, -1.7449448108673096, 0.2997722029685974, -1.791546106338501, -0.3085682690143585, 0.34958553314208984, -1.365088701248169, -0.7257466912269592, 0.5425826907157898, -1.2981431484222412, -0.4276081621646881, -0.5739852786064148, -1.0219850540161133, -0.33133214712142944, -1.7307264804840088, 0.44604629278182983, 2.0419442653656006, 0.9818089604377747, -1.2192964553833008, -0.1677211970090866, -0.7468535304069519, -0.06736503541469574, 0.38125619292259216, -0.3705997169017792, -0.6837068796157837, 0.7688261866569519, -0.058109186589717865, -0.20804114639759064, 1.9738373756408691, -0.08434376865625381, 0.34235185384750366, 0.6224233508110046, 0.7050369381904602, 1.2876262664794922, 0.49920493364334106, -1.536821961402893, 0.8762909770011902, 1.0389683246612549, -1.4383649826049805, 0.22076983749866486, 1.5127211809158325, -0.9942391514778137, 0.03906397894024849, -1.0519031286239624, 0.09643625468015671, 0.3792494237422943, 1.3220795392990112, -1.2509688138961792, -0.16507849097251892, -0.10270874947309494, -0.24350599944591522, -1.910253882408142, 0.10773256421089172, -2.267275094985962, 1.4535714387893677, 0.278592050075531, 0.02918401174247265, 0.23211652040481567, -0.4973655939102173, -2.641853094100952, -1.7439230680465698, -0.18515312671661377, -0.5695330500602722, 0.6296224594116211, 0.3441442847251892, -1.6339054107666016, 0.08028566092252731, 0.53397536277771, 0.7053285837173462, -1.2050350904464722, 0.2001020908355713, -0.043143775314092636, 0.6127545237541199, -1.9528931379318237, -0.6377058625221252, -0.7461302876472473, -1.6312153339385986, -1.0475577116012573, 1.0936774015426636, -3.5384435653686523, 1.1985154151916504, 0.5049247741699219, -0.38283395767211914, 0.5458987951278687, 1.1322078704833984, 0.23985953629016876, -1.902845859527588, -0.032463472336530685, 2.2118141651153564, 0.3308302164077759, 0.6879498958587646, -0.8149070143699646, -1.4959512948989868, -0.24149860441684723, 0.022152919322252274, 0.13405963778495789, 0.10098688304424286, -0.5049038529396057, -0.2492789626121521, 0.22002041339874268, -0.26444095373153687, 1.4826416969299316, -0.33493053913116455, -0.29131489992141724, 1.0313053131103516, 0.4948119819164276, -0.3400404155254364, 0.060649920254945755, -1.089092493057251, -2.1422054767608643, 0.3298009932041168, 0.1269330084323883, 0.36250564455986023, -0.9863507151603699, -0.5902378559112549, -0.4609557092189789, 0.46070724725723267, 1.7384960651397705, -0.7086448073387146, -0.734573483467102, 2.0801572799682617, 0.8665007948875427, -0.8178889155387878, 0.034345611929893494, 0.25198033452033997, 0.09298508614301682, 0.801577627658844, -0.3578774929046631, -0.4758386015892029, 1.3227910995483398, -0.636320948600769, -0.00991040375083685, -1.1672477722167969, 2.3246512413024902, 0.725598156452179, -0.1423906534910202, -1.383448839187622, -2.5021755695343018, -0.6791094541549683, -1.366084098815918, 1.1732637882232666, 1.4901318550109863, 0.03372909501194954, -0.0030240588821470737, 1.577338695526123, -0.5402893424034119, 2.185314655303955, 0.9385138750076294, -2.0714352130889893, -1.6465551853179932, 1.5027083158493042, -0.7779111266136169, -0.21661768853664398, 0.3282293975353241, 1.607932686805725, -1.1476342678070068, 1.273749828338623, 0.06383291631937027, 0.948965311050415, 0.32696932554244995, -0.43185946345329285, -0.06900990009307861, 0.14978580176830292, -0.5196854472160339, 1.0171579122543335, 0.9074978828430176, 1.5960570573806763, -0.31083518266677856, -0.2094539999961853, 0.5235587954521179, 0.11529015749692917, -0.23760977387428284, 0.18349727988243103, -0.10081078112125397, 1.7391928434371948, -2.54689884185791, 0.06781777739524841, -0.7485314607620239, -0.30157116055488586, 0.35403797030448914, 0.3366258442401886, -2.374659299850464, -2.779352903366089, 0.710862934589386, 0.5300410985946655, -1.8018311262130737, -0.9990501403808594, -1.175830602645874, -1.1678516864776611, -1.1988402605056763, -0.5663666725158691, 1.5355262756347656, 0.6437298059463501, 0.25233137607574463, -2.712982416152954, 0.6606871485710144, -1.0867584943771362, -0.9505572319030762, 1.717999815940857, 0.4313045144081116, -0.1194298192858696, 0.41275280714035034, -0.6948023438453674, -1.2070083618164062, -1.2865899801254272, 0.10898534953594208, 0.34445255994796753, -0.6509941816329956, 0.2887282073497772, -1.692765712738037, -0.672105610370636, 0.6410930752754211, 0.06822407245635986, 1.6268603801727295, 0.29117271304130554, -0.7073111534118652, -0.4714823365211487, -0.8666579723358154, 0.45783624053001404, 0.5346565842628479, 0.5650167465209961, -2.1843390464782715, 2.128577470779419, -0.3164190649986267, 0.7233056426048279, 0.6173468232154846, 1.0354667901992798, 0.6183077096939087, -1.2984658479690552, -0.9690119624137878, 1.198569416999817, -0.08225113153457642, 0.4392136037349701, -0.7880046367645264, 1.203534722328186, 1.1123762130737305, -0.4362424314022064, -1.7258580923080444, 0.7276318073272705, 1.5229517221450806, -0.5693922638893127, -0.03916703164577484, -0.32313546538352966, -0.1494879275560379, -0.46001678705215454, -0.4488089680671692, 0.14976970851421356, -1.4028570652008057, -1.359370231628418, -1.8650896549224854, 1.0191082954406738, -0.8405494093894958, -1.4763333797454834, -0.2845957279205322, -0.0005954967346042395, -1.1936745643615723, -0.44252902269363403, -0.3158096373081207, 0.46581530570983887, 0.5522343516349792, -1.1794757843017578, 0.3467908203601837, 1.0634665489196777, 1.9347517490386963, -0.5959283113479614, 0.4866292178630829, -0.313764363527298, -0.12751156091690063, -1.2994976043701172, 0.4176502823829651, -0.7169214487075806, 0.1987321376800537, 1.5925880670547485, 0.8752818703651428, -1.3301039934158325, -0.20791469514369965, -0.5625395774841309, -0.5190685987472534, -0.49005043506622314, -0.3523756265640259, 0.5288145542144775, 1.4272606372833252, -0.022256208583712578, -0.7785971164703369, -0.25478142499923706, -0.6289463043212891, 0.8984825015068054, 0.5539499521255493, 2.120413303375244, -0.010232372209429741, 0.11219010502099991, 1.6913210153579712, -0.5588408708572388, -1.0447378158569336, 1.1850361824035645, -0.30354979634284973, 1.6922661066055298, 0.9813410639762878, -0.28218740224838257, 1.2301877737045288, -0.3802475929260254, -1.146634578704834, -2.595632791519165, 1.7241955995559692, -0.5957724452018738, 0.9385957717895508, -0.17388616502285004, -0.7535334229469299, 0.31147289276123047, -2.5921332836151123, 1.0561041831970215, -0.6535627841949463, 1.5999864339828491, 0.7725734114646912, -1.7091857194900513, 0.5264605283737183, 0.9604337811470032, 1.9528535604476929, 0.2938973307609558, 0.31606924533843994, 0.1779322326183319, 0.890899121761322, 0.5439552068710327, -1.3902852535247803, 1.3681379556655884, -0.6188302636146545, -0.21584373712539673, -1.035729169845581, 0.42748069763183594, 0.14343132078647614, 0.4008854627609253, 0.5531414747238159, -0.6678500175476074, -0.31063657999038696, 0.22615592181682587, -0.7451733946800232, -0.1976384073495865, -0.2712934911251068, 0.015322420746088028, 1.3447842597961426, -0.5741894245147705, -0.3510681986808777, -0.8265196084976196, -0.24870900809764862, -2.1391351222991943, -0.5775959491729736, 0.4408451020717621, 0.6773862242698669, 0.8071950674057007, 0.6149880886077881, -0.31615322828292847, 2.3038980960845947, 0.5787559747695923, -0.19416165351867676, -2.3753821849823, 0.006937657482922077, -0.7151400446891785, 0.307854562997818, 0.20130103826522827, -0.8002805113792419, 1.08260178565979, 0.47765085101127625, -1.4473403692245483, 0.23597639799118042, -1.7392137050628662, -0.9766680598258972, -0.4851016402244568, -0.45968765020370483, -0.959450900554657, -0.12437187880277634, 0.29123222827911377, 1.675675392150879, -3.0057473182678223, 0.8241950869560242, -0.5687888860702515, -1.0984556674957275, -0.10791334509849548, 0.675110936164856, 0.5407313108444214, -0.4816661775112152, -0.34654688835144043, -0.23546338081359863, -0.27731674909591675, -0.9347070455551147, -0.6576703786849976, 0.46388036012649536, -0.7557119727134705, 0.8162506818771362, 1.972281813621521, 0.18505769968032837, 0.10346964746713638, -1.2442725896835327, 0.579264223575592, 0.9133400321006775, 0.09752623736858368, 0.12291114032268524, -0.36456289887428284, -0.8977311849594116, -0.5685725212097168, 2.2220818996429443, 1.6307377815246582, 0.37415966391563416, -0.28816574811935425, -0.0561830960214138, 0.7503699064254761, 1.0983787775039673, -0.17114077508449554, -0.7513225674629211, -0.6657510995864868, 0.23985859751701355, -0.7877998352050781, -0.48806819319725037, 0.689548909664154, 0.618789553642273, -2.3470053672790527, 0.42025622725486755, -1.2963604927062988, 0.8718997240066528, -1.5583256483078003, 1.9615843296051025, 0.0524255745112896, 0.6985794305801392, -0.4149642884731293, 2.2829928398132324, -1.222646951675415, -0.343259334564209, 0.9444771409034729, 0.4754751920700073, -1.7786428928375244, -2.478236436843872, 0.29895439743995667, 1.137847900390625, 1.409625768661499, -1.6988481283187866, -0.1029275432229042, -0.02153616026043892, -0.9489755034446716, 0.23304058611392975, -1.8220956325531006, -0.4638165831565857, -0.41342946887016296, 0.6380533576011658, 1.6403393745422363, -1.0807116031646729, -0.6120486855506897, -1.3598875999450684, -0.2406664937734604, 0.3030926585197449, 0.33624106645584106, -0.31261980533599854, 0.6301977038383484, -0.6361938714981079, -0.45646998286247253, 0.0053255497477948666, 0.6152598857879639, 0.016018183901906013, 1.2109684944152832, 0.17467404901981354, -0.4680487811565399, -0.8275928497314453, -0.7178438901901245, -0.5728892087936401, -0.01619575545191765, -2.1051011085510254, 1.335546612739563, 0.42535072565078735, 1.4974219799041748, -1.7622294425964355, 1.5388972759246826, 0.19067853689193726, 0.8573272228240967, -0.5465460419654846, 1.2497119903564453, -0.9218152761459351, 1.9080615043640137, -0.7900243997573853, -0.0791749358177185, 1.035987377166748, 0.03833203762769699, 1.5575292110443115, -0.3022971749305725, -1.6571130752563477, 0.3814302384853363, 0.6178260445594788, -0.5331008434295654, 1.3764019012451172, 0.49995771050453186, -0.37523141503334045, -0.46369779109954834, -1.4608187675476074, 1.1925384998321533, 1.589784026145935, -1.4349135160446167, 1.47292160987854, -0.18088817596435547, 0.6622409820556641, 0.7303327918052673, -1.6063647270202637, -0.31613656878471375, 0.7523826956748962, 1.3858914375305176, 0.5573969483375549, 1.6492999792099, -1.2623615264892578, 1.3084601163864136, 0.22351466119289398, 0.7483619451522827, 1.885252833366394, -0.06622454524040222, -0.6008416414260864, 0.4043869376182556, 0.006982055958360434, 0.006688866298645735, -0.159544438123703, -1.138855218887329, -0.20247326791286469, -1.548966646194458, -0.3883749842643738, -1.8919118642807007, -0.5023521780967712, 0.32042187452316284, -0.6958850026130676, -0.3602389693260193, 0.33849450945854187, -1.4090790748596191, 1.3026282787322998, 0.2560691237449646, -0.7620993256568909, 0.6886491179466248, 1.399938941001892, 1.708864688873291, 2.6248891353607178, -1.6621688604354858, 0.9035681486129761, 2.2264790534973145, -0.3497491776943207, 0.1669362336397171, 0.732853353023529, 1.9122676849365234, 0.2135612666606903, -1.0967901945114136, -1.2059688568115234, -1.6980308294296265, -1.9570175409317017, -0.16345301270484924, -0.2404022514820099, -0.41100576519966125, -0.45480480790138245, 0.8816048502922058, 0.1280960887670517, -0.8869739770889282, -1.735984206199646, 1.0127952098846436, 0.5300511121749878, -0.21357177197933197, -0.8291715979576111, -1.2343913316726685, -0.4600253999233246, 1.7738399505615234, -0.08878909796476364, -0.9013211727142334, 1.8010969161987305, 0.54063481092453, -0.4023093581199646, -1.5044969320297241, 0.11169400066137314, -0.21269255876541138, -0.551974356174469, -1.2222390174865723, -2.180629253387451, -0.16560907661914825, 1.176345705986023, 1.9331692457199097, -0.7531872391700745, 0.8543198108673096, -0.4313400387763977, 0.14843234419822693, 1.591491460800171, 0.041085656732320786, 0.03806218504905701, 0.7859759330749512, -0.8569713830947876, -0.8052811026573181, -0.33378422260284424, 0.06663113087415695, 1.0017321109771729, -0.43006935715675354, 0.08923707157373428, -0.45825234055519104, -0.20976954698562622, -1.1836484670639038, 0.1915665715932846, -1.1823694705963135, 2.333376407623291, 0.479444682598114, -0.35288986563682556, -0.4595086872577667, -0.7312333583831787, -0.16417397558689117, 0.8064125180244446, -1.911671757698059, -1.0871903896331787, 0.18851372599601746, -0.4417128264904022, 0.7813488245010376, -0.6764378547668457, 0.4816029965877533, -0.48630291223526, 0.4166529178619385, -1.3632476329803467, 1.5426499843597412, 0.17904074490070343, -0.8234527111053467, 0.39498966932296753, 0.922213077545166, -0.6877890229225159, -0.7766623497009277, -1.118607997894287, -0.809033215045929, 0.00665959483012557, 1.8931070566177368, -0.8011783957481384, 0.3794896602630615, -0.3556388318538666, -1.5301810503005981, -0.2444748878479004, -0.7586303353309631, -1.1919846534729004, 1.4138243198394775, -1.4671543836593628, -0.6583930850028992, 0.9757077097892761, 0.5951404571533203, -0.9656469225883484, 0.3231864869594574, -0.9763309955596924, -0.8364518880844116, -0.4871971607208252, 1.608416199684143, 0.2513654828071594, 1.4736907482147217, 1.2099329233169556, -0.04851221665740013, -0.4846819043159485, 1.3439356088638306, 1.4607404470443726, -1.3332990407943726, -1.1665441989898682, -0.021240413188934326, -0.6640907526016235, -0.9377918839454651, -1.5927339792251587, -0.045723699033260345, -0.8780019879341125, 0.2810593545436859, 0.2447025179862976, -0.11903810501098633, 1.036798357963562, -0.35952767729759216, 0.6544716954231262, 0.16583679616451263, -0.4813028573989868, 0.19455084204673767, -1.4576129913330078, 1.267296552658081, 0.19984634220600128, -0.6590614914894104, -0.8730332851409912, -0.3605816960334778, 1.785363793373108, -0.2693973481655121, 0.7269337177276611, 1.5149600505828857, 0.1432502418756485, -0.27065223455429077, -0.49438512325286865, -0.5055451393127441, 0.6586776971817017, 0.049046069383621216, -0.4014033377170563, 0.9469857215881348, -0.5225608348846436, -0.5490628480911255, 1.3122010231018066, 1.0948737859725952, -0.652888834476471, 0.15243135392665863, -1.133414626121521, -0.49484261870384216, 0.7598721981048584, 0.0884748324751854, 0.22460810840129852, -0.6702036261558533, 0.6962538361549377, -1.7587443590164185, -1.19715416431427, -0.9763438701629639, 0.0400213897228241, 0.5070131421089172, 0.5696001648902893, 1.2338653802871704, 0.2124955654144287, -1.5991367101669312, -1.1836421489715576, 0.5944069027900696, -1.0400784015655518, 0.5944297313690186, 0.278870552778244, 1.2247018814086914, -0.9153234362602234, -0.16458092629909515, -0.5019299983978271, -0.02512849122285843, 0.5900507569313049, 0.114592544734478, -0.14399662613868713, 0.4418507218360901, -0.0023770774714648724, -1.6862175464630127, -0.7571236491203308, 1.1133854389190674, -0.16688062250614166, 0.6944140791893005, 1.2250632047653198, 0.46239233016967773, -0.3880423903465271, 0.049591973423957825, 1.1130597591400146, 1.797641396522522, -0.10429375618696213, 0.15431565046310425, -0.47167059779167175, -0.04143832251429558, 0.4748302698135376, -0.05269560590386391, 0.6316940784454346, -1.089422583580017, 0.26975512504577637, 0.7084535956382751, 1.8756471872329712, -0.42250722646713257, -0.7473227977752686, 0.6551152467727661, -0.6610381007194519, 1.803847074508667, -0.09967408329248428, -0.5971361994743347, 0.10405335575342178, 0.5708630084991455, -1.5114173889160156, -0.8714647889137268, -1.8858009576797485, 1.7275474071502686, -0.9156836271286011, 1.6663304567337036, 0.5939075350761414, -0.27243614196777344, -0.023529792204499245, -0.6732821464538574, -0.019019175320863724, 1.2946678400039673, -0.06293287128210068, 1.0466949939727783, 1.8212170600891113, 0.3594076335430145], [-1.4289464950561523, -0.6488978862762451, 1.2319835424423218, -0.3799560070037842, 0.5031673312187195, 0.011710871011018753, 1.1106059551239014, -0.7875075340270996, -0.09958803653717041, 0.03595816344022751, 0.7227341532707214, -0.2812170386314392, 0.9860357642173767, -0.3452979326248169, 0.1388835608959198, 0.007178420200943947, 1.2719470262527466, -0.8672582507133484, -0.025179872289299965, -0.1346370130777359, 0.059422463178634644, 1.4702742099761963, -0.901690661907196, -0.48817169666290283, -0.10536200553178787, -0.5050756335258484, 1.3657339811325073, 1.5205610990524292, -1.4970202445983887, -0.13580535352230072, -1.0252482891082764, -1.008368968963623, 0.02618466503918171, 0.24336102604866028, -0.5972067713737488, 0.4451098144054413, 0.008563913404941559, 1.0328929424285889, -1.0189791917800903, 0.6001231074333191, 0.40814974904060364, -0.16070832312107086, -1.1101040840148926, -3.334563732147217, -1.131980061531067, -0.4026162624359131, 0.005866252817213535, 0.865689754486084, 1.4815701246261597, -2.488739013671875, -0.13133391737937927, -1.3348591327667236, 0.6643399596214294, 0.36039435863494873, 0.12641488015651703, -0.39737775921821594, -0.04128911346197128, 0.9870441555976868, 0.9890820980072021, 0.7502966523170471, 0.034762557595968246, 0.4318212568759918, -0.8785597681999207, 0.8703380227088928, 0.9873026013374329, -0.12157692015171051, 0.9686315655708313, -0.6072959899902344, 0.5387033820152283, 1.1592466831207275, -0.4339894652366638, 0.81048184633255, -0.3538224697113037, -1.3298381567001343, 0.13240885734558105, -1.0406707525253296, -1.2975404262542725, -0.42730003595352173, -1.6247522830963135, 0.5635213255882263, 0.45864248275756836, 0.908967137336731, -1.1152466535568237, 1.04098379611969, 0.7139784693717957, 1.2589876651763916, -1.1825462579727173, -1.3292629718780518, 0.45176416635513306, 0.3737170696258545, 1.061434030532837, -0.6938626766204834, 0.7855172157287598, 1.1654157638549805, 0.4447023570537567, -0.5861756205558777, 0.277791291475296, 0.009454645216464996, 1.1586042642593384, -0.8866509795188904, 0.3723790943622589, -0.36416178941726685, -0.05990707129240036, -2.0583035945892334, -0.980523407459259, 0.2610223591327667, -1.5034146308898926, -0.0216690581291914, 0.36258745193481445, -1.1391994953155518, 0.35608622431755066, -2.3089873790740967, -1.2063453197479248, -0.6237346529960632, 0.5026870369911194, 0.3562077283859253, 1.5922752618789673, -0.10257591307163239, -0.6737167239189148, -1.273785948753357, 1.3646847009658813, 0.5786343812942505, 1.262841820716858, 1.5653718709945679, 0.09362509846687317, 0.7508485317230225, 1.0319139957427979, -1.386987328529358, 0.01202667411416769, 1.1050785779953003, 0.4049912691116333, -1.9477883577346802, -1.1178733110427856, 1.2663514614105225, 0.2775331735610962, -0.6626613736152649, 0.5727320313453674, -1.3630298376083374, -1.784061074256897, 0.7548990249633789, -0.661697506904602, 0.291329562664032, 0.7480251789093018, -0.7326104044914246, -1.1968327760696411, -1.3833513259887695, 0.8786572217941284, -0.3387433588504791, 1.0999854803085327, -0.6735031008720398, 0.390579491853714, -0.03644293546676636, 0.8131983876228333, -0.1349267214536667, -0.024797732010483742, 0.3144838809967041, 0.06068849191069603, 0.24133709073066711, 0.7505648136138916, 0.48573100566864014, 0.7766134738922119, -0.7168899178504944, 0.9997079372406006, 0.7922048568725586, 1.2855082750320435, 0.7023410797119141, 0.7234023213386536, 0.709983766078949, 0.9072579741477966, 0.049472469836473465, 1.283362627029419, -0.24040180444717407, -1.3991466760635376, -0.21677514910697937, -0.4276728332042694, -1.034385085105896, -0.2183067798614502, -0.2145271748304367, -0.18256008625030518, -0.29982495307922363, 1.6748472452163696, -0.8014866709709167, -1.0271782875061035, 0.05222991108894348, 0.46046966314315796, 0.5074901580810547, -1.0067251920700073, 1.3760257959365845, -0.08120621740818024, 0.6530877351760864, -0.4442053437232971, -1.0058948993682861, 1.1552921533584595, -2.190025806427002, -0.6768060326576233, -0.4854799807071686, 1.2075318098068237, -0.30787762999534607, -0.9370846152305603, -1.4166738986968994, -0.8409363031387329, 1.7694369554519653, -1.086340308189392, 0.09950093924999237, -1.1430314779281616, -0.9083477854728699, -0.053829602897167206, 1.7619541883468628, 0.166107639670372, 0.8984670639038086, -1.7536412477493286, -1.6000056266784668, 0.20790235698223114, 0.37102580070495605, -0.6084655523300171, -0.994740903377533, -1.8030345439910889, -0.9593015909194946, -0.9072104692459106, -0.8186445236206055, 0.21120043098926544, 1.4679681062698364, -0.5692511200904846, 0.5729971528053284, 1.3283030986785889, -0.7994328737258911, 2.0192437171936035, 0.5997313857078552, 2.384577751159668, -2.076462507247925, 1.672792911529541, -0.5537901520729065, -0.6727797985076904, -0.6337182521820068, 1.004130244255066, 0.48331621289253235, -0.40697360038757324, -0.7652724981307983, 1.0472675561904907, 2.2634711265563965, 0.7781789898872375, 0.28765490651130676, -1.6074756383895874, -0.02920333668589592, 0.7601544857025146, 0.6817681789398193, -0.5011742115020752, 1.627583384513855, -1.4352591037750244, 0.38709551095962524, 1.5787855386734009, -0.9525855183601379, 1.7485548257827759, 0.7265962958335876, -0.8961743712425232, 0.37492918968200684, -0.28542405366897583, 2.0039358139038086, 0.08657795935869217, 1.2515946626663208, -0.588655412197113, -0.2814284861087799, 0.8720153570175171, -1.0511173009872437, 0.4351823031902313, 2.303964853286743, -0.2732258439064026, 0.7132573127746582, 0.08857062458992004, 1.0597894191741943, -2.1146388053894043, 1.8427608013153076, 1.0878207683563232, 0.15500979125499725, -0.7850765585899353, 2.188990831375122, 0.3507445454597473, -0.36319875717163086, 1.2672258615493774, 0.5801514387130737, 0.7544211745262146, 1.959141731262207, 1.4918475151062012, 2.6932153701782227, 1.7259047031402588, -0.5030490756034851, 0.15101860463619232, -0.9904985427856445, -0.00044179210090078413, -1.0300379991531372, 0.029104456305503845, -0.09218011796474457, -0.8855753540992737, 1.2209044694900513, -0.5386122465133667, 0.11774512380361557, 0.8966162204742432, 0.6342775821685791, 0.7608224749565125, -1.3149176836013794, -1.8184418678283691, -0.009114290587604046, 0.5332236886024475, -1.122223973274231, -0.8272011280059814, 1.5350887775421143, -0.2696029245853424, 0.09010817855596542, 0.24750077724456787, -0.7042859792709351, 1.1498618125915527, 0.8318981528282166, 1.3372951745986938, 1.6692895889282227, 0.5131422877311707, -1.607284665107727, 0.8488547205924988, -0.19441461563110352, 0.620987594127655, -0.029113003984093666, 0.3092779219150543, -0.37817198038101196, -1.768938422203064, 0.1894868016242981, 0.734700620174408, -0.7490464448928833, -0.2071899026632309, -0.9045408964157104, 0.4819832444190979, -0.7052462697029114, -1.318145751953125, 0.28809651732444763, 0.978015124797821, 0.5378244519233704, 0.5516927242279053, 2.459209442138672, -0.6354880928993225, -1.0857521295547485, 0.5701289176940918, -0.3220258951187134, -2.0469772815704346, 1.5767192840576172, 0.08522336930036545, -0.373485267162323, 1.6654481887817383, -0.1322156935930252, 1.8553614616394043, -0.721824049949646, -1.1950303316116333, -1.4440593719482422, -0.617553174495697, -0.6710591316223145, 0.3372362554073334, -0.7156283259391785, -1.116493821144104, 0.2854248285293579, 0.6165914535522461, -0.03809914365410805, -0.567321240901947, -0.33248016238212585, -0.1918351948261261, 0.02795003540813923, -1.0314120054244995, -0.7799631953239441, -0.9684104919433594, 0.433573842048645, 1.8078712224960327, 1.3227852582931519, -1.841799020767212, 0.45431041717529297, -0.5746816992759705, -1.6860167980194092, 1.0472924709320068, 0.0988720953464508, 0.1079103872179985, -0.08357539027929306, 1.1548105478286743, -0.11646466702222824, -1.5719349384307861, 0.25824999809265137, -0.23991256952285767, -1.984985589981079, 1.2655272483825684, 1.2197082042694092, -1.726465106010437, -1.1295392513275146, -0.5574920177459717, -0.0382888987660408, 0.39747920632362366, 1.8936415910720825, 0.5040016174316406, 1.608407735824585, -1.6077452898025513, 1.0298632383346558, 0.4224368631839752, -1.0740370750427246, -0.4822315573692322, -0.21650603413581848, 1.4088315963745117, -0.6351677775382996, -0.5591377019882202, -0.20101352035999298, -0.6154314875602722, 0.20080731809139252, 0.769869863986969, -0.9103165864944458, 0.8806037306785583, 1.0247925519943237, -1.0379314422607422, 0.34687188267707825, -1.1502912044525146, 0.6173347234725952, 0.44525763392448425, -1.0160341262817383, 0.3283728361129761, -0.5416340231895447, -0.9040668606758118, -0.12681816518306732, -0.8743021488189697, 0.31879064440727234, 0.13738568127155304, -0.07687383145093918, -0.06485728919506073, -1.4267953634262085, -0.7465270757675171, 0.3583207130432129, 0.7845627069473267, 1.5031336545944214, 0.048253681510686874, 0.9887954592704773, 0.5719032287597656, 0.04265940934419632, 0.7144219875335693, -0.2743445336818695, 1.1935315132141113, -0.24845290184020996, -1.4997830390930176, 0.4010399878025055, -0.5425930619239807, 0.16182096302509308, 0.9366201162338257, -0.9155953526496887, -1.0452218055725098, 0.11002733558416367, -0.9518466591835022, -0.9508703947067261, 0.2221492975950241, 0.4682837128639221, 0.7154742479324341, -0.16880877315998077, -0.2917775809764862, 0.14667686820030212, 2.127579689025879, 1.7384597063064575, 0.4564630389213562, 0.9399096965789795, 0.7312207221984863, 0.015403306111693382, 0.5251510739326477, -0.18578431010246277, -2.245255470275879, 0.6713832020759583, 1.8260246515274048, -1.0619688034057617, 0.5904805660247803, -0.27159497141838074, 1.456258773803711, -0.37597984075546265, 1.6405783891677856, -1.964924931526184, -0.517124593257904, 1.4197925329208374, -0.6590599417686462, 1.0802620649337769, -1.5026763677597046, -0.08588642627000809, -0.4453403055667877, 1.2349624633789062, 1.327630877494812, -0.679082453250885, -0.14345823228359222, 0.8941775560379028, -0.12100078910589218, -0.6712819933891296, 0.008136667311191559, 0.9454320073127747, 0.14544737339019775, -1.032230019569397, 0.690368115901947, 0.5412845015525818, -0.9450612664222717, -0.14273278415203094, 2.429330348968506, -0.6308552622795105, -0.3503398001194, -0.9935369491577148, -1.5205739736557007, 1.246484637260437, -2.0662498474121094, -0.4196149706840515, -1.2660069465637207, -0.9684831500053406, -0.13900797069072723, -0.17609542608261108, -0.2739248275756836, 0.181938037276268, -0.4052269458770752, 1.2119086980819702, -0.873455286026001, 0.7454423904418945, -0.42163166403770447, 0.03885505348443985, 0.7436805367469788, 0.659079372882843, -0.19976653158664703, 0.604284405708313, 2.3315811157226562, 1.7622199058532715, 0.37106776237487793, -0.545219361782074, -0.7788392901420593, 0.8664295673370361, 1.213332176208496, -0.09359415620565414, -0.48669999837875366, -0.8489861488342285, -0.6519449353218079, -0.5979028344154358, -0.4720085561275482, 1.2495999336242676, 1.6081748008728027, 1.3167927265167236, 1.5118296146392822, -0.6719686985015869, 0.7290637493133545, -0.4079609215259552, 0.49655207991600037, -0.8037465810775757, 1.2869807481765747, 1.1784757375717163, 1.379999041557312, -0.8923437595367432, -0.5235962271690369, 0.11259555071592331, -1.0383442640304565, 0.922146201133728, -0.06069156527519226, -0.5806958675384521, -0.8767812252044678, -0.39411646127700806, 0.24123604595661163, 0.4483586549758911, 1.0640970468521118, 0.16319558024406433, 0.6488841772079468, 0.3901521563529968, 1.1211464405059814, -1.0165878534317017, -1.317326545715332, 2.1110944747924805, -2.867513656616211, -0.9783389568328857, -0.8513522744178772, -0.8735879063606262, -1.1681307554244995, -0.6913811564445496, -1.8384859561920166, 0.533136248588562, -0.32726752758026123, -0.9178136587142944, 0.8091593980789185, 1.050849199295044, -0.07026983797550201, 0.03476583585143089, -0.5392376184463501, -0.44815394282341003, -1.7872580289840698, 0.1991947442293167, 0.9002307057380676, -0.05556175112724304, 0.0536406971514225, -1.983424186706543, 0.18290738761425018, 0.4693893492221832, -0.49538272619247437, -1.8955572843551636, -0.3068733215332031, 1.2684035301208496, -1.6435678005218506, 0.66170334815979, 0.5213404893875122, 0.2162381112575531, 0.09712730348110199, -1.2191064357757568, -0.24596595764160156, 0.41946670413017273, 0.3016342520713806, -1.4886977672576904, 0.8309199213981628, 1.0046699047088623, 0.9189573526382446, -0.887231707572937, -0.6495238542556763, 0.702143669128418, 3.052546501159668, -0.5006533861160278, -0.32899603247642517, -0.3191562294960022, -1.3487578630447388, -0.31652945280075073, -2.283824920654297, -0.7508033514022827, 1.8825565576553345, 0.3878936469554901, -1.2148977518081665, 0.45322635769844055, 1.1977860927581787, 1.0241457223892212, 0.14602069556713104, -0.470876008272171, -0.5425809025764465, 0.4297381341457367, 0.2730686664581299, -0.8499775528907776, 0.11396457999944687, -1.5803344249725342, 0.540015459060669, 0.4700373411178589, 0.3889372646808624, 1.3916548490524292, 0.22058920562267303, -1.1209437847137451, 1.0750840902328491, 0.3307015001773834, -0.33120623230934143, -0.5640645027160645, 0.23086969554424286, -0.6316121220588684, 0.6809779405593872, 0.5091379880905151, 1.1619830131530762, 0.5292467474937439, -0.5395042896270752, 0.9914115071296692, -0.8670605421066284, 1.4489727020263672, -1.4826699495315552, -0.9303624629974365, 1.4389827251434326, -0.01713212952017784, -1.008331537246704, -1.0342649221420288, 0.08389481902122498, 2.7713022232055664, -0.44897446036338806, -0.24915210902690887, 0.8431813716888428, -1.6303437948226929, -0.1914745271205902, 1.0851199626922607, -0.013810502365231514, 0.8585922718048096, -0.22565464675426483, -1.8650257587432861, -0.37355729937553406, 1.4558380842208862, -1.727274775505066, 0.49997565150260925, -1.6069114208221436, 0.9903333783149719, 0.17688947916030884, 1.2209947109222412, -1.3923760652542114, -1.8785552978515625, 1.3295952081680298, -0.2817707359790802, -0.34511733055114746, 0.1829957365989685, 0.6340958476066589, 0.27400916814804077, 0.8129754662513733, -1.100886344909668, 0.05561366304755211, -1.8038976192474365, -0.4999757409095764, -1.51626455783844, 0.6312952041625977, 1.3234697580337524, -0.7070119380950928, -0.6356459856033325, 0.11235419660806656, -1.3366400003433228, -0.07232773303985596, -0.4475052058696747, -0.5773980617523193, 0.6194810271263123, -0.8176591992378235, 0.21854880452156067, 0.555949866771698, 1.0194010734558105, -0.9463579654693604, 0.801394522190094, 0.2447730302810669, -1.60651695728302, -0.43982169032096863, 0.7343119978904724, 0.4038766920566559, -0.9063652753829956, -0.07643085718154907, -1.46394944190979, -1.057744026184082, -0.9970334768295288, -0.6315547227859497, -0.9700083136558533, -1.5674152374267578, 0.7932745814323425, 0.35532137751579285, -0.9733186960220337, -0.3906071186065674, -0.9533336758613586, 1.0488187074661255, 0.5085118412971497, 1.4345817565917969, 2.1544525623321533, 0.2461090087890625, -0.8201125860214233, 0.7713664770126343, -0.2831456661224365, 1.4895172119140625, 0.608985424041748, -2.311988353729248, -1.9241514205932617, 0.660334587097168, -1.0254369974136353, -0.8445641398429871, 0.8150933384895325, -1.651036262512207, 0.04927217587828636, 0.9023623466491699, 1.0439566373825073, -0.40757259726524353, -0.3778704106807709, 0.28280001878738403, 0.17783014476299286, 0.8866243362426758, -1.05210280418396, -2.0943357944488525, 1.0516389608383179, -0.17257872223854065, -0.18931090831756592, 1.0758583545684814, -0.08494967222213745, 0.6269397735595703, 1.0902010202407837, -0.7639346122741699, 1.086355447769165, -0.6526567935943604, -0.5980921387672424, 1.1765462160110474, -0.18069519102573395, 0.22640469670295715, -0.7360692620277405, -0.2674749195575714, 0.7317104339599609, 0.08215612918138504, -0.9873420596122742, -0.3747195601463318, 1.1475729942321777, 0.8022139072418213, -0.6817830801010132, -0.7221110463142395, 0.2603428363800049, 0.0665970966219902, 0.914585292339325, -1.3104071617126465, -1.2701817750930786, 1.7911018133163452, -0.3861469626426697, -1.586245059967041, 0.267267107963562, 1.682887077331543, 0.8006487488746643, 0.021113615483045578, -0.5447192788124084, -1.5115196704864502, -0.22285497188568115, 1.536507487297058, -0.8321340084075928, 0.7251628637313843, -0.5583247542381287, -1.0174922943115234, -1.2932159900665283, 0.27391937375068665, 0.040383078157901764, 0.08107779920101166, -0.04071500897407532, -1.0577551126480103, -1.3320684432983398, 1.1207857131958008, 0.8957552313804626, 1.3203223943710327, -0.12926235795021057, 1.1893410682678223, 1.211207389831543, 0.054233092814683914, -1.8339191675186157, -1.9663043022155762, 0.588854193687439, 0.475841760635376, -1.8162462711334229, 0.48118165135383606, -0.14383529126644135, -0.35226792097091675, -0.7453364729881287, 0.6675000786781311, -1.8070027828216553, -1.2493677139282227, -0.9701992273330688, 0.9550818204879761, -0.42795905470848083, -1.3038805723190308, 0.05604789778590202, -0.9849997758865356, -2.2973806858062744, 0.33596423268318176, -0.7232699394226074, -0.11907576024532318, -0.4735795259475708, 0.12784041464328766, -1.8508492708206177, 0.6585992574691772, -0.5972042679786682, 1.811655044555664, 0.6190732717514038, 1.0891584157943726, -0.5882562398910522, 0.5086064338684082, 0.2875383496284485, -0.5375120639801025, -0.7600249648094177, 0.15532585978507996, -1.8146179914474487, 0.5337585806846619, 1.3012384176254272, 0.4900951385498047, -0.9946757555007935, 1.500145673751831, 0.2793348431587219, 0.19353178143501282, -1.674592137336731, -0.22442662715911865, 0.3535007834434509, 1.1213817596435547, -0.3138328194618225, 1.7904126644134521, -0.2336609661579132, 0.4383159279823303, -0.9134294986724854, -1.450645089149475, -0.4263582229614258, 0.20201711356639862, 0.7553144097328186, -0.2056288868188858, -0.19706611335277557, 0.41584497690200806, -0.013921386562287807, 0.8670276999473572, 1.047745943069458, 1.2028042078018188, -0.03545663505792618, -0.11464467644691467, 1.845292091369629, 1.3279659748077393, 0.9374180436134338, 1.6883567571640015, 1.178312063217163, 1.7632603645324707, 1.1277358531951904, -0.9784224033355713, 0.8487768769264221, -1.6612696647644043, 0.05966462194919586, -0.6633577942848206, -0.10425695031881332, -2.7238359451293945, -0.8588923811912537, -1.3543312549591064, 0.45344972610473633, -1.3608758449554443, 0.7397369742393494, 1.4423264265060425, -0.2134372740983963, 1.1703006029129028, 1.1163159608840942, -1.2290384769439697, 1.4665802717208862, -2.2840402126312256, -0.43031632900238037, 1.0421456098556519, 0.46690115332603455, 1.163912296295166, -1.913125991821289, 0.7069720029830933, -0.24545437097549438, 2.077579975128174, 0.8205167651176453, 0.6551013588905334, -1.0898879766464233, 0.11305802315473557, -0.550595760345459, -1.1500887870788574, -1.7527698278427124, 0.8803421854972839, 2.078277587890625, -0.3312719762325287, 0.2451067417860031, -0.33722883462905884, -0.8557767868041992, -0.4510856866836548, 0.4810827374458313, 0.6222183108329773, -0.8182815313339233, -1.3511297702789307, 0.5895798206329346, 2.4294023513793945, -1.269816517829895, 0.04587457701563835, -1.1018435955047607, 0.13815714418888092, 0.19721105694770813, -0.772059440612793, -0.8879023194313049, -0.5767055749893188, 0.030254311859607697, -0.5082207918167114, -2.3018157482147217, 0.7900097966194153, -0.2738185226917267, -1.4309148788452148, 0.09523866325616837, -0.08034492284059525, -2.1464297771453857, -0.44692033529281616, 0.8344451189041138, 1.8132648468017578, -0.02986239455640316, -0.7606756687164307, -2.537816047668457, -0.5641558170318604, 0.016650773584842682, -3.0024163722991943, 0.26028621196746826, -0.726677417755127, 0.12367390841245651, -0.6330829858779907, 1.4581209421157837, -1.0303045511245728, 0.3777439296245575, 0.07971799373626709, -0.6062126755714417, 1.0783419609069824, -0.33619776368141174, 0.7698768377304077, 1.2962710857391357, -1.265997052192688, -1.2449852228164673, 0.1345871239900589, 0.7203846573829651, 0.38063594698905945, -0.02073722705245018, 1.7024357318878174, 1.7124707698822021, 0.2705571949481964, -0.7725359797477722, -0.7159470915794373, -0.24431267380714417, 1.0456914901733398, -2.9792001247406006, 0.2569476068019867, 0.5363738536834717, 1.788752555847168, -2.9408345222473145, 1.956222653388977, -1.1618107557296753, 1.7545415163040161, 0.7153496146202087, 1.2567561864852905, 0.5615845918655396, -1.791221261024475, -1.2103480100631714, -0.052524384111166, -1.241407871246338, -0.0018179254839196801, -0.7232661247253418, 0.477862149477005, -0.9786397218704224, -0.685065746307373, 0.9591906666755676, -0.3188971281051636, 0.7540780901908875, -0.635131299495697, 2.0578060150146484, 0.07003653049468994, -0.34618160128593445, 0.7033296823501587, -1.731368899345398, 0.44478198885917664, 0.7729333639144897, 0.7708068490028381, 0.9539170861244202, 0.02749093621969223, -0.11783186346292496, 0.9199810028076172, -1.025710105895996, 1.9159672260284424, -0.6651480793952942, 0.9187306761741638, -0.21363724768161774, -0.24094060063362122, 0.685083270072937, -0.7619240283966064, -0.684830367565155, 0.9453224539756775, -1.3601652383804321], [-0.7521387338638306, 0.3442998230457306, -0.9084969162940979, 1.9316200017929077, 1.4617586135864258, 0.42009225487709045, -0.7038002610206604, -2.140495777130127, 0.9235446453094482, 0.10242369025945663, 1.0293904542922974, 2.802889585494995, -0.5691843032836914, -1.2029258012771606, -0.027156196534633636, -1.704057216644287, 0.36215928196907043, 1.16020929813385, 2.6512529850006104, -0.6696664690971375, -0.25243449211120605, 0.11232326179742813, -0.08117394149303436, 0.16823506355285645, 1.933935523033142, -0.6171560883522034, 0.12506386637687683, -0.7862754464149475, -0.3218124806880951, -0.7857626676559448, -0.5620215535163879, 1.153886318206787, 1.564652681350708, -0.3524351418018341, 1.0430638790130615, 0.3322197198867798, 1.5340417623519897, 0.2719280421733856, -0.2775803208351135, 0.8331379294395447, -0.12481097131967545, 1.3001691102981567, 1.1827185153961182, 0.07429961115121841, -1.1627986431121826, 0.31174004077911377, 0.3282410800457001, -1.8336598873138428, 0.13301220536231995, -0.880398690700531, -0.16811120510101318, 0.26497647166252136, -0.49706000089645386, -0.880608081817627, -1.5503407716751099, 0.7897513508796692, 0.17297327518463135, -0.06471908092498779, 0.15621693432331085, 0.006137095391750336, -0.7315570712089539, -0.5025784373283386, 1.1698576211929321, -0.07893708348274231, 0.3256457448005676, 0.636024534702301, -0.6369273662567139, 1.5077215433120728, 0.7072873115539551, 1.3564279079437256, 0.38722771406173706, -0.78452467918396, -0.10158193856477737, -0.736417829990387, -0.6453181505203247, 1.139868140220642, -0.21809802949428558, 0.45903995633125305, 0.8614081144332886, -1.8797237873077393, 0.33323463797569275, -0.6864790916442871, -1.1461073160171509, 0.4571513533592224, 1.6665589809417725, -1.0793497562408447, 0.709038496017456, 0.838938295841217, 0.7699877619743347, -0.7359749674797058, 1.0608943700790405, 1.0878788232803345, 1.8535685539245605, -0.3576842248439789, -0.3505989909172058, 0.7500358819961548, 2.4584238529205322, -0.970064103603363, -0.4355183243751526, 0.32227787375450134, 2.267913579940796, 2.864901542663574, -0.5088089108467102, 0.16588044166564941, 1.125523567199707, -1.1426806449890137, -0.7448734045028687, 2.030808925628662, 2.196749448776245, -1.7510968446731567, -1.1360305547714233, 0.20682795345783234, -0.7541249394416809, -1.8841584920883179, 1.376196265220642, 0.6078953146934509, 0.5829812288284302, -1.071855068206787, -0.36988136172294617, -1.349012851715088, -1.523970603942871, -1.4836769104003906, -1.4201923608779907, -0.6537676453590393, 1.9021906852722168, -0.12738989293575287, 0.33293089270591736, -1.175531268119812, -0.40040919184684753, -1.6566694974899292, 1.1969003677368164, -0.3684954047203064, -1.0071862936019897, -0.9672011733055115, 1.0309209823608398, -0.6700940132141113, -0.1461128443479538, 0.9773600101470947, 1.4446601867675781, 0.7099983096122742, 0.3636854588985443, -0.7764137387275696, -2.035656213760376, -0.36760327219963074, -0.24899521470069885, 0.4639534056186676, 0.6956145763397217, 0.1475788652896881, -0.6788759827613831, -0.4359653890132904, -0.762205183506012, 1.93962562084198, -0.8255255818367004, 0.4991415739059448, 1.0413926839828491, -0.34557273983955383, -0.8442448377609253, 0.16721703112125397, -0.2074674814939499, -0.2708885073661804, -0.8007920980453491, -0.9578315019607544, 1.7951295375823975, 0.08089084178209305, -0.32491445541381836, 1.1268165111541748, 0.951448380947113, 0.7907615303993225, -0.4063897430896759, 0.6509406566619873, -1.8361053466796875, 0.11170511692762375, -0.6113147735595703, 0.16641300916671753, 0.3920094072818756, 1.0981900691986084, -1.5810660123825073, 0.23711508512496948, -2.0687978267669678, 0.42315250635147095, 0.2230842262506485, 0.14882563054561615, -0.897955060005188, -1.0063085556030273, 0.6814984679222107, 0.4142448902130127, -0.34640586376190186, -0.34200042486190796, 1.3733057975769043, 0.08529496192932129, -0.7216199636459351, -1.6651966571807861, 0.08550074696540833, -0.7358505725860596, -1.371904969215393, -0.5040830373764038, -0.40677499771118164, -0.7267023921012878, 0.30288079380989075, -1.3760623931884766, -0.5008686184883118, -1.6908799409866333, 1.5148053169250488, 1.2772493362426758, -0.9875548481941223, 0.7355170845985413, -0.6385316252708435, 1.5972436666488647, -0.7711935043334961, 1.1251343488693237, 1.1983259916305542, 1.5063284635543823, -0.795370876789093, 1.2213250398635864, -0.4489651620388031, -1.48908531665802, -0.012602110393345356, 1.3887414932250977, -0.38729456067085266, -0.5430702567100525, 0.17321930825710297, -0.19885796308517456, 0.4285464584827423, -0.8962008953094482, 2.56072998046875, -0.08257449418306351, 0.790118396282196, 1.6805559396743774, -1.5229727029800415, -1.6916759014129639, 0.9778448939323425, 0.3801051676273346, 0.21057280898094177, -1.684418797492981, 0.9278711080551147, 2.0231642723083496, 1.305196762084961, 0.7280853390693665, 2.2794690132141113, -1.5615041255950928, 0.9669840335845947, -0.43400588631629944, -0.16413001716136932, 0.6836352348327637, 0.9253762364387512, 0.9503453969955444, 1.5945965051651, -0.35420000553131104, 1.2230244874954224, 0.319714218378067, 0.5044030547142029, 1.8690555095672607, -0.2595325708389282, -0.4554192125797272, -0.8233157396316528, -0.2506622076034546, 0.1751747876405716, -1.679545521736145, -1.2286601066589355, 0.8156580328941345, -0.8187391757965088, 0.06832165271043777, 1.9478018283843994, 1.1017438173294067, 1.6128653287887573, -0.6328116655349731, 0.3082781732082367, -1.8441280126571655, -1.1806321144104004, -2.5602355003356934, 2.27893328666687, -0.42663848400115967, 1.8825364112854004, -0.9636070132255554, 0.9578064680099487, -0.4291086792945862, -0.3470193147659302, -0.261966735124588, -0.4094903767108917, 1.8878062963485718, 1.1230218410491943, 1.1962844133377075, -0.5499464869499207, 0.36711469292640686, -0.39788246154785156, -1.3706873655319214, 1.1780436038970947, 1.5733150243759155, -0.2726178467273712, -0.018591970205307007, -0.7131345272064209, 0.5074231028556824, -0.6872486472129822, -0.5853182673454285, 0.4815254211425781, -0.11821334064006805, 0.34482496976852417, -0.060318611562252045, -0.40694481134414673, 0.7346112132072449, 0.8489139676094055, 0.9797449707984924, 1.7746047973632812, 0.31391626596450806, -1.33499014377594, 0.048053570091724396, 0.003975383937358856, 0.2164926379919052, -0.8778617978096008, -0.27829283475875854, 3.26114821434021, 2.6797866821289062, -0.033250756561756134, 1.6313016414642334, -1.0795844793319702, -0.5121791958808899, 0.3933999538421631, 0.2795219421386719, 0.964279294013977, 0.9319015741348267, -0.2615008056163788, -0.1112583726644516, -0.5564525723457336, -1.260807752609253, 0.919767439365387, -0.9216427206993103, 2.0170540809631348, 0.34148308634757996, 1.083649754524231, 1.1718835830688477, -0.17184704542160034, 1.7035249471664429, 0.6234393119812012, -0.258845716714859, 1.4233514070510864, 1.290596604347229, 0.7487190961837769, 0.4497096538543701, 1.691090703010559, 0.6144695281982422, 1.4510807991027832, -0.3698425590991974, 0.3626864552497864, -2.6611521244049072, -0.04690510407090187, -1.1042559146881104, 0.9731158018112183, -0.5836965441703796, 1.8135876655578613, -0.11612579971551895, 0.6943575143814087, -0.31663334369659424, -0.41797876358032227, 0.43090566992759705, 0.790464460849762, -0.7825159430503845, -0.17638446390628815, 0.46590515971183777, 0.8594956398010254, -0.029284143820405006, 0.3227679431438446, -0.8804497718811035, -0.5227441191673279, -1.009087085723877, 0.8089228272438049, -0.12722665071487427, 0.8288403749465942, -0.4711383581161499, 0.8580610752105713, 0.5996926426887512, 0.8652338981628418, -0.8531503677368164, 0.571864128112793, -1.1842070817947388, 0.23079843819141388, -1.3777201175689697, -0.0775189995765686, -1.0682246685028076, 0.13362282514572144, 1.3461360931396484, -1.8224583864212036, -0.31726154685020447, -0.9661962389945984, 1.094556450843811, -0.65000319480896, -1.0922753810882568, -0.348039448261261, 0.9951233863830566, -0.8075169324874878, 0.4604799449443817, -0.40439221262931824, 1.5765652656555176, 1.1977730989456177, 0.23129382729530334, 0.15244366228580475, -1.3607598543167114, 0.5588505268096924, 0.41313937306404114, -1.2312448024749756, 0.20458984375, -1.5269556045532227, -1.473984956741333, -0.07782655209302902, 0.3475688397884369, -1.1388986110687256, 0.6177799701690674, -2.0395829677581787, -1.113505482673645, -0.4866340160369873, -0.2396533191204071, -0.03495630994439125, -1.390779733657837, -0.7085868120193481, 0.13954992592334747, -0.22689111530780792, -0.30038097500801086, -0.14722485840320587, -0.47341397404670715, -0.7891349196434021, -0.5263516902923584, 0.15663526952266693, 0.5188273787498474, -0.1939772516489029, -2.110093355178833, 0.44014567136764526, -1.1948729753494263, 1.4538333415985107, -0.0074215843342244625, 0.47304633259773254, 0.7107084393501282, -1.2672704458236694, 0.8415462970733643, 1.4479670524597168, -2.329380989074707, -0.3653965890407562, 1.4735499620437622, -1.0800905227661133, 0.0733414813876152, -0.17037178575992584, -0.8187269568443298, 1.7505093812942505, -0.36304545402526855, 1.429264783859253, -0.6576692461967468, -1.108472466468811, 1.0137261152267456, -0.7007955312728882, 1.0890417098999023, 2.2815253734588623, 0.06707228720188141, 0.49000126123428345, 0.43995726108551025, -0.7770472764968872, 0.04566037282347679, -0.3823513388633728, 1.5517170429229736, -0.07276732474565506, 0.5445359349250793, 1.0102030038833618, 0.19467423856258392, 0.05599503219127655, 0.9157675504684448, -0.22604581713676453, 0.6996392011642456, 0.10510864853858948, 1.2388873100280762, 0.9981998801231384, 0.4790128469467163, -0.24981379508972168, 0.34524714946746826, 0.9924682974815369, 0.29024139046669006, 0.22307677567005157, 0.7502778172492981, -2.080498695373535, 0.2784423232078552, 0.4935204088687897, 0.45211929082870483, 1.4480218887329102, -1.0380287170410156, 0.6146603226661682, -1.5296440124511719, 0.9663447141647339, 0.5978103280067444, -0.722514808177948, -0.5986422896385193, 0.453548789024353, 1.2276450395584106, 1.4179177284240723, -1.2177232503890991, -1.6077978610992432, 0.9410848617553711, -1.0179557800292969, 1.0670175552368164, 0.5146638751029968, 0.6929868459701538, -0.03269895911216736, 1.6225968599319458, -0.41248273849487305, 0.8447137475013733, -1.331627607345581, 1.0796653032302856, -2.130620002746582, -0.04521079361438751, -0.09030367434024811, 0.24549569189548492, -0.12516407668590546, -0.046936217695474625, 0.2694677412509918, 0.7478383183479309, 0.5080997943878174, 0.28928592801094055, 0.283612459897995, -1.3212437629699707, -0.8402789831161499, -0.8196188807487488, -1.513695240020752, -0.48094987869262695, -0.7208427786827087, 0.19660547375679016, 0.7509514689445496, -1.6379060745239258, -0.026740077883005142, -1.0558249950408936, 0.021239930763840675, -0.3100849986076355, -1.2362618446350098, -0.1292000263929367, -2.0677196979522705, -0.6847946047782898, -0.09947267174720764, 0.07182463258504868, -0.06840719282627106, 1.7254027128219604, -1.4013530015945435, -0.3606756627559662, 0.10049650073051453, 0.7996171712875366, -0.5868504047393799, 1.4215975999832153, 0.1720946729183197, 0.39279332756996155, -0.8284584879875183, 0.11272764205932617, -0.6183358430862427, -0.7717637419700623, -0.12289655953645706, -0.03660992532968521, 0.3590582013130188, -1.4140642881393433, 0.27676331996917725, -0.5176110863685608, -0.37497738003730774, 0.042976148426532745, -1.6010487079620361, -0.9810847043991089, -0.09624951332807541, 0.553966224193573, 1.7752132415771484, 0.17718054354190826, 0.8129969835281372, -0.10099490731954575, -1.0069373846054077, -0.7991443872451782, -0.02515803463757038, -1.0590659379959106, 0.5807346701622009, -0.2132374495267868, 2.421398878097534, 3.1136200428009033, 0.7249568104743958, -0.08731578290462494, 0.49275636672973633, -0.46103668212890625, 0.28873494267463684, -1.3592939376831055, 0.06786898523569107, 0.544819176197052, -1.160724401473999, -1.246529221534729, -0.3829091191291809, -0.3999599516391754, 0.7282383441925049, 1.877300500869751, -0.747367799282074, -0.8535906672477722, -1.1447466611862183, -0.207549050450325, -0.30207177996635437, 1.5067492723464966, -0.4263962507247925, -0.03816401958465576, -0.27860552072525024, 0.06442461907863617, 0.5891384482383728, 1.5584264993667603, 0.1014629527926445, -0.18712186813354492, -1.071751356124878, -1.2285008430480957, -1.0879907608032227, -0.22382061183452606, 0.14104458689689636, 0.12008199095726013, 1.037800908088684, -0.5647958517074585, -1.0271788835525513, 1.1921658515930176, 0.4674557149410248, -0.06133945286273956, 0.4703460931777954, -1.0096813440322876, -0.38912320137023926, -0.49002590775489807, 0.40072473883628845, -0.7854125499725342, 1.430175542831421, -1.4108332395553589, 0.479451060295105, -1.364009141921997, -0.9647589325904846, -1.4347047805786133, -1.8456095457077026, -0.48876699805259705, 1.306550145149231, -0.37518811225891113, 0.030328113585710526, -0.15823376178741455, 0.8957903385162354, 1.0385481119155884, -0.4481200575828552, -0.999447226524353, -0.8120887875556946, -0.766747772693634, -0.7262287735939026, 0.1729733794927597, -0.39732909202575684, 1.1820672750473022, 0.8519805073738098, -0.5310218930244446, 0.20823118090629578, 1.2372255325317383, 0.2563563883304596, 1.1939910650253296, 0.002625811845064163, -0.41280731558799744, -0.46761250495910645, -0.12819921970367432, 0.9170592427253723, -0.11236781626939774, -0.06567812711000443, -1.0979564189910889, 0.7151715755462646, 1.3944118022918701, -0.4563300907611847, -0.37239131331443787, -1.4800286293029785, -0.19966264069080353, 0.8750460147857666, -2.8590188026428223, -1.7265803813934326, -0.1935638040304184, -0.5690004825592041, 0.07700826972723007, 0.12814152240753174, 0.6981724500656128, -0.198897585272789, 0.18638314306735992, -0.2706252634525299, -0.9314805865287781, 0.498140424489975, 1.3686033487319946, 0.02731972187757492, -1.448103666305542, 0.41334185004234314, 1.2681834697723389, 0.4871705174446106, 0.08934857696294785, -2.8397936820983887, 0.9371659159660339, 0.11743615567684174, 0.7624810338020325, -0.7305744290351868, -1.315657615661621, 0.23160320520401, -1.0452871322631836, 1.9293217658996582, 1.6776416301727295, -0.2853054404258728, 0.19649115204811096, -0.006068036425858736, -1.6971787214279175, -1.5344445705413818, -0.645695149898529, -1.2597180604934692, 0.312916100025177, 1.3443814516067505, 2.332756996154785, -0.3296215534210205, 0.15430329740047455, -0.5683375597000122, 0.8390960693359375, 0.2135208696126938, -0.5680815577507019, -0.5587591528892517, -0.2483072131872177, -0.7472180724143982, 0.32878246903419495, 0.5155234932899475, -1.0635581016540527, -1.308763861656189, -1.437368392944336, 0.6406999230384827, -0.23321260511875153, 0.3275029957294464, -0.7481653094291687, -0.2840178608894348, 1.2291778326034546, -1.3829550743103027, 0.08518220484256744, -0.4854205846786499, 1.6508084535598755, -0.5523751378059387, 1.8857816457748413, -0.3042507469654083, 0.47713345289230347, -0.8815895318984985, 1.224684715270996, -0.2269323468208313, 0.08256355673074722, -1.2137285470962524, 0.2326362580060959, 0.9465042352676392, -0.21472904086112976, 1.271357536315918, -0.7753254175186157, 0.07841690629720688, -0.16236542165279388, 0.4912700653076172, -0.5844033360481262, -0.8021503686904907, -0.7717651724815369, -0.4018267095088959, -0.12766674160957336, -0.5216298699378967, 1.337542176246643, -0.6417362093925476, 0.43863457441329956, 2.0287411212921143, -0.5839468240737915, 0.8100665211677551, 0.02091148868203163, -0.9927769303321838, 0.5717326402664185, -0.694126307964325, 0.22976531088352203, 1.4030705690383911, 0.6825911402702332, -0.28203675150871277, 0.6400231719017029, 0.5477741360664368, -0.10265996307134628, -1.692184567451477, -1.0613980293273926, -0.009386057034134865, -0.8469528555870056, 0.9446446895599365, 0.49327629804611206, 0.7965793013572693, -0.526145339012146, -2.234970808029175, -0.5700175762176514, -0.8917349576950073, -0.28746509552001953, -0.14365437626838684, -0.41635993123054504, -1.781432867050171, -1.2604948282241821, 1.4086833000183105, 0.9763712882995605, 0.8863334059715271, -0.05768816918134689, -0.40137821435928345, -0.014395264908671379, 1.759560227394104, 0.34280070662498474, 0.949851393699646, -1.5044814348220825, -0.8042582869529724, 0.9408043622970581, -0.9164064526557922, -0.1322387158870697, -0.7458498477935791, 1.2957266569137573, -1.9040945768356323, -1.0459413528442383, -1.3801600933074951, -2.388566732406616, 0.9828475117683411, -0.17501302063465118, -0.5932007431983948, 0.6178202629089355, 0.7433912754058838, -0.13780485093593597, 2.031580924987793, -0.4439406394958496, 0.7367253303527832, -1.1031800508499146, -1.4939320087432861, -0.1791541874408722, -1.4065030813217163, 2.2405717372894287, 0.26598992943763733, -0.11449521780014038, 3.044537305831909, -0.052016839385032654, 0.2620544731616974, -0.9026628136634827, 0.08170174807310104, 0.05328512191772461, 0.12797945737838745, 0.3691210150718689, 0.04433527588844299, 0.3953704535961151, 0.42956113815307617, 0.00025218428345397115, -0.6752389073371887, 0.7520449161529541, 0.5617514848709106, -0.7354022860527039, -0.19135785102844238, -0.3352538049221039, -0.4265775978565216, -0.2610068917274475, 0.5236760973930359, -1.5668624639511108, -0.8516400456428528, -0.916748046875, -0.2579123079776764, -0.7649515867233276, -0.7906375527381897, 0.8850739002227783, -0.870137631893158, 1.0809365510940552, 0.1229623332619667, -0.49369800090789795, -0.6083379983901978, -0.317128449678421, -0.4909374713897705, -1.0792309045791626, 0.4198625981807709, -0.5161407589912415, -1.0636389255523682, -0.5748916268348694, -0.508966863155365, 0.44064652919769287, -0.41971683502197266, 0.3758626878261566, 0.5975514054298401, -1.796347975730896, -1.1051570177078247, 1.5802721977233887, 0.677838921546936, -1.926804780960083, -0.4828048646450043, 1.2101309299468994, -0.19203633069992065, -0.17438210546970367, 0.7435908913612366, 0.038788940757513046, -0.09843680262565613, 0.938495397567749, 0.8191317915916443, 1.5167300701141357, 0.6336299777030945, 0.29599037766456604, -2.536639928817749, 0.8137181997299194, 1.2857939004898071, -0.5675550699234009, 0.026977023109793663, 0.48195841908454895, -0.9940004348754883, -1.527127742767334, 1.1035412549972534, -0.7070865035057068, 0.6403777003288269, -0.6200085878372192, -0.10690657049417496, 0.24864980578422546, -0.4153115153312683, 0.9490441679954529, -0.5692263841629028, -1.079057216644287, -0.31066256761550903, 0.4504351317882538, -0.22932031750679016, -2.027768850326538, 0.14933012425899506, 0.30810168385505676, 1.1006090641021729, 0.08182944357395172, -0.3737832009792328, 0.15413814783096313, 0.3407803475856781, -1.0122538805007935, -0.3748675286769867, -0.0018588457023724914, 0.3259250819683075, 0.07281820476055145, -0.36910077929496765, 0.006563067901879549, 0.10811977833509445, -0.5403602123260498, -0.7041048407554626, -2.9842777252197266, 1.7612316608428955, 0.5896857380867004, -0.9975227117538452, -0.8315057158470154, 0.8910191655158997, 1.557116150856018, -1.1169999837875366, -0.027835268527269363, 2.0189859867095947, -0.8285007476806641, -0.25446876883506775, 0.633090615272522, 0.5741586685180664, -0.8473591208457947, -0.7284207940101624, -0.9949318170547485, 1.9518537521362305, -0.11043529957532883, 0.44151952862739563, 0.5515373349189758, 0.9405504465103149, -0.5168744325637817, -0.25336021184921265, 0.537443995475769, 0.4029272198677063, -1.6035492420196533, -0.20603284239768982, -1.9841526746749878, -0.7555104494094849, 0.13185833394527435, -1.608028769493103, 0.5409279465675354, 2.202911853790283, 0.5290308594703674, -1.413480281829834, 0.019519653171300888, 0.5631387829780579, -0.8652380704879761, 0.45407944917678833, 0.3376610279083252, 0.1477268785238266, -1.2615547180175781, -0.5003844499588013, -0.857404887676239, 1.315632939338684, 0.9776275753974915, 0.03636891767382622, -0.39224180579185486, -0.38177114725112915, -1.1940847635269165, 1.018691062927246, 1.2469099760055542, -1.182844877243042, 0.7489914894104004, -1.1371594667434692, -1.2174955606460571, 1.1081877946853638, 0.6581633687019348, -0.7568315863609314, -0.3499338924884796, 1.091719150543213, 1.391111969947815, 0.23119328916072845, -0.13538311421871185, 0.7682838439941406, -1.282232642173767, -1.3987083435058594, 0.20917999744415283, 1.3307236433029175, -0.2406553030014038, -0.5757086873054504, 0.5728968381881714, 0.02760060504078865, -0.4200604557991028, 0.6002137660980225, 0.2587010860443115, 1.1514332294464111, 0.38210198283195496, 0.6181032061576843, 0.7554861903190613, -0.5541113615036011, -0.6459153294563293, 0.09228824079036713, 0.15226960182189941, 1.883514642715454, 1.8738356828689575, -0.8922255635261536, -0.7151013612747192, 0.6969802975654602, 1.1795315742492676, -0.7340347170829773, 1.5622235536575317, 0.9708371162414551, -1.0242974758148193, -0.583013653755188, -1.6756333112716675, 1.2099982500076294, 0.6675798296928406, -0.37185052037239075, -1.4712507724761963, 0.9887176156044006, -2.25451922416687, -0.1130005344748497, -1.2195796966552734, 0.6604053974151611, 1.0260258913040161, -0.5898609757423401, 0.6246814727783203], [-0.754401445388794, -0.9085332155227661, -0.37319228053092957, -0.5237402319908142, 0.9683332443237305, -1.1355043649673462, -0.5889643430709839, -0.18221142888069153, -0.9160094857215881, -0.1516077071428299, 0.8371812105178833, -0.25527113676071167, 0.7392483353614807, 1.0477555990219116, -0.24291636049747467, -1.1154260635375977, -1.1627899408340454, -0.5770888328552246, 0.9428006410598755, 0.036262959241867065, -0.47916603088378906, 0.40074074268341064, 1.3147878646850586, 0.08958036452531815, 1.220118522644043, 0.6696427464485168, -0.5468337535858154, 0.9723848104476929, 0.7836006283760071, 1.3728772401809692, 0.6094481945037842, -0.38896864652633667, -1.1030280590057373, -0.8705489039421082, -0.02577170357108116, -1.314327597618103, -0.06382469832897186, 0.7766814827919006, 1.3994970321655273, -0.23845219612121582, 2.79742169380188, 0.8767335414886475, 2.3983945846557617, 1.0513290166854858, -0.29420581459999084, 0.3330526649951935, -1.7990347146987915, 0.15164203941822052, -1.0089575052261353, -1.2316431999206543, 0.7005078792572021, -1.0149489641189575, -0.4888301491737366, -0.9561916589736938, -1.8575587272644043, -1.563416838645935, -2.048523187637329, -0.5444987416267395, 1.3872629404067993, 1.3659487962722778, -0.04222533106803894, 0.20369374752044678, -0.598207950592041, 0.7682857513427734, -0.09344585984945297, 1.8311444520950317, -0.7883796095848083, 0.1252286285161972, 0.9861853718757629, -0.714103102684021, -0.2383112907409668, -0.44209370017051697, 0.036584075540304184, -0.5391920804977417, 0.6651948690414429, -1.0035159587860107, 1.3620434999465942, 0.7380015254020691, -0.43426254391670227, -0.717191755771637, 0.36149731278419495, 0.4450501501560211, 0.7165603637695312, -0.9474454522132874, -0.5140495896339417, -0.27615731954574585, 0.10023993998765945, -0.1769258677959442, -1.1626948118209839, -0.05840524286031723, -0.6060489416122437, -0.7925884127616882, 0.05566003918647766, -3.426927328109741, -0.2126034051179886, 1.1084885597229004, 0.08291244506835938, 0.9021240472793579, 1.476830244064331, -0.25127482414245605, -2.406634569168091, 0.3882724940776825, -1.6755410432815552, -0.08243405073881149, 0.7182201743125916, 2.3599853515625, 1.51017427444458, -2.204540967941284, 0.7724690437316895, 1.1410679817199707, -0.003876901464536786, 0.5626538395881653, -0.01104732509702444, -0.5830975770950317, -0.9746613502502441, 1.4164645671844482, 0.31993523240089417, -1.046726942062378, -0.5147193670272827, -1.1362885236740112, 1.7748572826385498, -0.03074718825519085, -0.6791349053382874, 0.9746869802474976, 1.3691426515579224, -0.8279463052749634, -0.9577381610870361, -0.7324066162109375, -0.5412542819976807, 1.433467984199524, 0.3573700785636902, -0.9109415411949158, 0.4028036296367645, -0.8882666230201721, -0.19683480262756348, -0.40301018953323364, 0.2560320496559143, 0.8314782381057739, -1.4649794101715088, 0.4743061661720276, 1.1690605878829956, -0.11633146554231644, 1.1059998273849487, -0.3850274980068207, 0.8433523774147034, -0.758410632610321, -0.3077987730503082, -0.9154806733131409, -1.1370413303375244, -0.5513027906417847, 0.3288553059101105, -0.31350231170654297, -2.5815412998199463, 0.5983257293701172, 0.13803312182426453, 0.6349971890449524, -2.5327954292297363, -0.5292050242424011, 1.5048837661743164, 0.25783753395080566, -0.24527385830879211, -1.171176791191101, -1.3238754272460938, 0.22859682142734528, 0.6962775588035583, 0.7830354571342468, -1.3589216470718384, -1.342692494392395, -0.9209555983543396, 2.6618244647979736, 0.41320571303367615, -0.3612343370914459, -1.18119478225708, -1.8901422023773193, 1.3453388214111328, -0.366392582654953, -0.014275460503995419, -1.2930748462677002, 0.5637739896774292, -0.5147908329963684, 1.0586804151535034, -0.44966891407966614, 1.5729668140411377, -0.4802985191345215, -0.2627110779285431, 0.2556467652320862, 0.40635767579078674, -0.13633406162261963, 0.4883219301700592, 1.603351354598999, -1.659994125366211, 0.06881300359964371, 0.775480329990387, 0.9752107858657837, 0.06709282845258713, -2.184943914413452, -0.4480009973049164, -0.5191398859024048, -2.054853677749634, 1.248710036277771, 0.6870410442352295, 0.11137913167476654, -1.177648901939392, -1.8101223707199097, -0.5815652012825012, 1.7751857042312622, 0.12768812477588654, -0.050890762358903885, 2.0552079677581787, -0.00568393524736166, 1.6059558391571045, 0.8267602324485779, -0.39669838547706604, 1.612817406654358, -0.2595212161540985, -0.6759007573127747, 1.820308804512024, -2.296358585357666, -1.2403299808502197, 0.1253761351108551, 0.24418416619300842, -2.5977771282196045, 0.9787144064903259, 0.21856659650802612, -0.6720834374427795, 0.40682855248451233, 0.10317779332399368, -0.7340973615646362, 0.656561017036438, -1.0422362089157104, 0.70408695936203, 0.9722675085067749, -0.3759446442127228, -0.635736346244812, -1.067046046257019, -1.689414381980896, 0.11555518954992294, 0.0859609842300415, 1.811757206916809, 1.0823683738708496, 1.3604938983917236, 0.03611323982477188, 0.8716073036193848, -0.009838663041591644, 1.6586683988571167, -1.318692922592163, -0.9933809638023376, -0.43622061610221863, 0.8029962778091431, 1.0002617835998535, -0.24732787907123566, -1.0058989524841309, 0.3600383996963501, -2.2375693321228027, -0.49637168645858765, 0.8011247515678406, 1.674554467201233, -1.955823540687561, 0.41624918580055237, 1.1573377847671509, 0.269695907831192, 2.0457308292388916, 0.9198551177978516, 0.08193102478981018, -0.6547257900238037, 0.1356971114873886, 0.3217649757862091, 0.41983163356781006, -0.11136084794998169, 1.3542596101760864, 1.7323640584945679, 0.6239011883735657, 0.3803309202194214, -0.3927990794181824, 0.4752216637134552, 1.1977946758270264, 1.7742737531661987, 1.0127726793289185, -1.0172758102416992, -0.4721757769584656, 0.3288567066192627, -0.568299412727356, 0.9784513115882874, 0.42362862825393677, -1.4981670379638672, -0.9219334721565247, -0.04285576194524765, 0.5425108075141907, -0.12494911998510361, 0.1983199268579483, 0.4320875406265259, 1.071905255317688, 0.6071475148200989, 0.849388837814331, -0.3046315312385559, -1.1551748514175415, -0.5793625712394714, 0.1625654697418213, -0.6136471033096313, -0.16695447266101837, -0.08705185353755951, 0.7012223601341248, -0.7873551249504089, -0.8789071440696716, -1.032855749130249, 1.399380087852478, 0.0807710662484169, 0.7436178922653198, 1.583193063735962, -0.03576696291565895, 0.3976825475692749, -1.8366416692733765, -0.8943652510643005, 0.8017393946647644, 1.454568862915039, -0.046396300196647644, 0.1311028152704239, 0.047871820628643036, 1.535652995109558, -1.5586379766464233, -0.6426393389701843, -0.5367379784584045, -1.9795138835906982, 0.21262390911579132, -0.13356968760490417, -0.6954243183135986, -0.8390308022499084, 1.2357134819030762, -1.335151195526123, -0.44253087043762207, -0.18691402673721313, -1.767311692237854, -0.8002936840057373, 0.6263570189476013, 1.0911883115768433, 1.1483317613601685, 0.5584464073181152, -3.1996045112609863, 1.353177785873413, 0.7614968419075012, 0.9480635523796082, -0.8456587791442871, 0.6901494264602661, 0.3966076672077179, -1.5335057973861694, -1.306097388267517, -1.5836029052734375, -0.2956986725330353, -1.2535403966903687, -0.4132853150367737, 0.7917352318763733, 0.09641964733600616, -0.2899755835533142, -0.6088709235191345, -0.0650542750954628, -0.026215923950076103, 0.7854279279708862, -0.8830644488334656, -1.8509941101074219, 0.6289567947387695, -1.8457120656967163, -2.1874349117279053, -1.0551872253417969, -1.4198319911956787, -0.37570616602897644, 0.23686903715133667, 0.396074116230011, -1.6452808380126953, -1.5406749248504639, 1.3937901258468628, -0.6237630844116211, -1.8819377422332764, 0.3033161759376526, 0.8398818373680115, 1.8570247888565063, 1.6495883464813232, 0.1427709460258484, 1.7275930643081665, 0.709177553653717, -0.8650001287460327, 0.5326225757598877, 0.9895532131195068, -0.37822574377059937, 1.6981337070465088, 0.6156231760978699, 1.0137231349945068, -0.8204016089439392, 1.2004858255386353, 1.226141095161438, 0.06392839550971985, -1.2904905080795288, 0.6943564414978027, -0.9479283690452576, -1.2174676656723022, 1.5303035974502563, -0.21950547397136688, -0.7817116975784302, -0.507076621055603, -1.2966217994689941, -0.7945199012756348, -1.4446914196014404, -0.9555304050445557, -2.215702772140503, 0.782296359539032, -0.5611836910247803, -0.5797534584999084, 1.5082918405532837, -0.28682762384414673, -2.315115451812744, 0.3204333782196045, 0.982467532157898, 0.44088542461395264, -0.027608953416347504, 1.5125203132629395, -0.06739530712366104, -0.7358360290527344, 0.4165719151496887, 2.6313908100128174, -0.9554023742675781, -0.3484150171279907, 0.8828731179237366, 0.965217113494873, -0.4914451837539673, -1.134036660194397, -0.20945417881011963, -0.3092682659626007, -0.6079270243644714, -1.5874916315078735, -0.5205039381980896, -2.438297748565674, 1.1624492406845093, 0.14833077788352966, 1.364684820175171, -0.3765054941177368, -0.9869686365127563, -0.3704962134361267, 1.2820470333099365, 1.1819432973861694, 0.3669123947620392, -0.4945559501647949, 1.2432551383972168, 1.4016121625900269, -0.4536304175853729, 0.013113138265907764, 1.1467417478561401, 0.09389784187078476, 0.40341565012931824, -0.9023702144622803, 0.13574081659317017, -1.3386383056640625, -1.0795001983642578, -2.433809518814087, -0.1789567768573761, 1.6827641725540161, 0.41993018984794617, 0.9334480166435242, -0.3470127582550049, -0.07416724413633347, -1.0692906379699707, 0.9360918402671814, 0.7967845797538757, 1.188424825668335, 0.40613147616386414, 1.9354921579360962, -0.03201953321695328, 1.7772306203842163, 0.22037024796009064, -0.31035682559013367, 1.2234625816345215, -0.8569115400314331, -0.3661651313304901, 0.4043978154659271, 0.23375143110752106, 2.3221614360809326, -0.866456151008606, -0.051054760813713074, 0.2036428153514862, -2.326263666152954, -0.742188036441803, -1.8566882610321045, -0.7010725736618042, -0.7779896259307861, 1.1911373138427734, -2.0816762447357178, 0.3433484137058258, 2.0435774326324463, -0.3814952075481415, 0.4143020212650299, 0.36855626106262207, -0.7204878330230713, -0.36150112748146057, -2.161059617996216, -1.8234225511550903, 1.768126368522644, 1.4510058164596558, -1.6045089960098267, -0.4944908320903778, 0.34433066844940186, 1.2310210466384888, -0.3955312967300415, -0.31847402453422546, 0.7907525300979614, 0.21598106622695923, -0.08879180252552032, -0.43738511204719543, -0.9752625823020935, -1.2375009059906006, 1.3948818445205688, -0.14395670592784882, 0.19899716973304749, -0.993387758731842, 1.0106555223464966, -1.8171119689941406, 0.1843852996826172, 0.5435982346534729, 0.6404507160186768, 1.6687449216842651, -1.3339437246322632, 0.3766416907310486, -0.027868472039699554, -0.5332164764404297, -1.154293417930603, -0.950672447681427, 0.7972738146781921, -0.308711439371109, 0.9834002256393433, -1.341489553451538, -1.734521746635437, -0.7605296969413757, -1.6447417736053467, 0.25636667013168335, 0.5036518573760986, -0.6423928737640381, -1.7903894186019897, 1.0616322755813599, -0.4408736526966095, -0.19174015522003174, 0.17140012979507446, -0.17352919280529022, -0.3763682544231415, 0.31034812331199646, -1.0878100395202637, -0.31230321526527405, 0.532978892326355, 0.44497430324554443, -0.33436253666877747, -0.6604729890823364, 0.9575940370559692, -1.2810574769973755, 0.8511467576026917, 0.7270542979240417, -0.07084790617227554, 0.1963338404893875, 0.06912261247634888, -1.378425121307373, -0.20661909878253937, 1.411940574645996, 0.045316752046346664, 0.6058221459388733, -1.287631869316101, 0.06730178743600845, 0.09144527465105057, -0.46710532903671265, 0.4694710671901703, -0.4217754304409027, 0.5310047268867493, 0.11481138318777084, -1.436387300491333, -0.5257589221000671, 0.16367857158184052, -0.5908240675926208, -1.1122589111328125, -1.4421268701553345, -0.4350901246070862, 0.8529744744300842, 1.2756661176681519, 0.4963042438030243, -0.9695546627044678, 2.488123893737793, -1.1809349060058594, -0.6961483359336853, -1.6727699041366577, 1.805493712425232, -0.9896913766860962, 1.3700928688049316, 0.701533854007721, 0.223308727145195, -1.3257559537887573, 0.3349135220050812, -0.031766537576913834, -0.8930570483207703, -1.4916861057281494, -0.11708490550518036, -0.8685641288757324, 0.26882094144821167, -1.8353601694107056, 0.7750522494316101, 0.9189764261245728, -1.2025184631347656, 1.4079011678695679, -0.3103318214416504, -0.1230456605553627, -0.5451357364654541, 0.9637698531150818, -1.2374378442764282, 1.2139198780059814, -0.3555612862110138, 2.2806789875030518, -0.7532999515533447, -0.7406890988349915, -0.029243865981698036, -1.6190099716186523, 0.3297921419143677, 0.07867264747619629, 0.8347169756889343, 0.33723804354667664, 0.40804046392440796, -1.9771970510482788, -0.16607147455215454, 1.102551817893982, 0.3059881925582886, -1.044175386428833, -0.0075270687229931355, 0.7070202231407166, -0.3244644105434418, -0.34044408798217773, -0.2867705225944519, -1.2857085466384888, 0.2661263942718506, -0.8708769083023071, 1.5031007528305054, 0.3432362973690033, 0.06351916491985321, -1.5531245470046997, 0.8507537245750427, -1.2449719905853271, -0.9538237452507019, -0.14711205661296844, -2.0076324939727783, -1.67536461353302, -0.40393128991127014, -0.11440892517566681, -0.7464905381202698, 0.44039836525917053, 1.3351590633392334, -2.831089496612549, -1.2081336975097656, -1.5502232313156128, 1.0673633813858032, -0.6718477606773376, -0.03869831934571266, 0.7669587731361389, -0.023582860827445984, -1.0693299770355225, 1.6429743766784668, -1.063071370124817, 0.7559508085250854, 0.9015457630157471, -1.2935826778411865, -0.17051416635513306, -2.597696304321289, -1.6491886377334595, 1.228015661239624, 0.8718220591545105, -0.4474982023239136, 1.2911053895950317, -0.0012556385481730103, 0.23419015109539032, 0.6737380623817444, 2.8121438026428223, -0.9491413831710815, 0.347549706697464, 0.19733253121376038, -0.8003618121147156, 0.6802452802658081, -1.0867220163345337, -0.6548323035240173, 1.4652544260025024, -0.06891769915819168, -2.2840425968170166, 2.2685976028442383, 0.8490014672279358, -0.8808708786964417, -0.17430263757705688, 0.21872960031032562, 0.2221931666135788, -0.6162594556808472, 0.25893229246139526, -0.08722186088562012, -0.379623681306839, -1.0593585968017578, -1.1871004104614258, 0.9015886783599854, -0.7566516399383545, -0.282413125038147, 0.2755604684352875, 0.8519759178161621, -0.9444544911384583, -0.048962902277708054, 1.4043304920196533, -1.5552383661270142, 0.5765131711959839, -0.38674047589302063, -1.4814940690994263, 0.44200557470321655, -1.3301008939743042, -0.9779791235923767, -0.21286684274673462, -1.3571197986602783, -1.1415187120437622, -0.20301564037799835, 1.1635056734085083, 0.40520399808883667, 0.3275602459907532, -0.3262755572795868, 1.4115407466888428, 0.05105390399694443, -1.0999586582183838, 1.087760329246521, 0.1419316530227661, 0.28902965784072876, -1.6659036874771118, 1.635767936706543, 1.8728561401367188, 0.7042972445487976, 2.256615161895752, -1.6224806308746338, 0.91534024477005, -1.3161629438400269, -1.0961161851882935, -0.2684761881828308, 0.577979564666748, -0.3735092878341675, 0.31064489483833313, -0.6555400490760803, 0.8562774062156677, -0.3788059651851654, 0.6188331246376038, 1.398385763168335, 0.29335832595825195, -2.6055421829223633, 0.289562851190567, -0.6705517172813416, -1.4200987815856934, 2.5212621688842773, 0.22033894062042236, 1.0020185708999634, -1.3935188055038452, 0.2791811227798462, -0.3603804111480713, -0.31099656224250793, 1.0647529363632202, -0.9544931054115295, 1.129404067993164, -0.026652388274669647, 0.3543463349342346, -0.7856311798095703, 0.24875132739543915, -1.2574028968811035, -0.5706643462181091, 1.1496398448944092, 0.6334279179573059, 0.7566366195678711, 0.3650467097759247, -2.5065674781799316, -0.28497833013534546, 0.4196068048477173, 0.5184496641159058, 0.4242059886455536, -0.8930641412734985, -0.6433862447738647, 0.7918007969856262, -0.3105781674385071, -0.8794782161712646, 0.8602803349494934, -0.7989258170127869, -0.4431973397731781, 0.12214849889278412, 0.7496553063392639, -0.00734149944037199, -0.0298860352486372, -0.6710842251777649, -2.1109888553619385, 0.38018080592155457, 1.7805190086364746, 0.32008811831474304, 0.5675498843193054, 0.35037460923194885, 0.45997950434684753, -1.0535857677459717, 0.8373607993125916, 0.10548670589923859, -0.5975237488746643, -0.4358716607093811, -1.393754243850708, 0.7707630395889282, 0.11653468012809753, 0.5713459849357605, -1.2470941543579102, -0.5195350050926208, -1.1218026876449585, 0.2585211396217346, -0.9281429648399353, -0.7398762702941895, 0.4915008842945099, -1.065430998802185, -0.29470178484916687, -0.8376074433326721, -0.006979112979024649, -0.2756291329860687, 2.2279608249664307, 0.556402325630188, -1.480278491973877, -0.0025129891000688076, 0.7345191836357117, 0.23497334122657776, 0.45712000131607056, -0.7500050663948059, -0.9154214262962341, -0.15128014981746674, -1.9227280616760254, 0.837632954120636, 0.5521438121795654, 0.2374451905488968, 0.8761875033378601, -0.17618979513645172, -0.8590627908706665, -0.7315182089805603, 0.12817971408367157, -1.8195476531982422, 1.1074731349945068, 1.43360435962677, -0.9474582672119141, 0.40698012709617615, 0.31678250432014465, 0.5353109836578369, -1.5840106010437012, 2.6849284172058105, 1.5839545726776123, -2.198376417160034, -0.08810793608427048, -0.01048765517771244, -0.7942964434623718, -0.29117581248283386, -0.4198033809661865, 0.930359423160553, -2.600844383239746, 1.1286921501159668, -0.45241406559944153, 0.1899842917919159, 0.801963210105896, 1.3145182132720947, 2.109016180038452, 0.3824424743652344, -0.7150270342826843, -0.2782963812351227, 1.8575738668441772, -0.8240005373954773, -1.4561868906021118, -1.9371644258499146, -0.7804492712020874, -0.036509208381175995, 0.4628012776374817, 0.3948277235031128, -0.895132839679718, 0.5929280519485474, -1.0940375328063965, 0.4361235499382019, 1.7969539165496826, 0.40428853034973145, -0.1883377581834793, 0.27710986137390137, -0.48620009422302246, -0.36392003297805786, 0.43183350563049316, -0.5901007056236267, -1.0872288942337036, 0.6040617823600769, -1.4151175022125244, 1.8229888677597046, 0.2893209457397461, 0.6319547295570374, -1.2583545446395874, -0.4311693608760834, -1.09488844871521, -0.4877398908138275, 0.34665435552597046, 0.39805692434310913, -1.3664826154708862, 1.6006872653961182, -1.303174614906311, -1.528314471244812, 0.38372156023979187, 1.1032212972640991, 0.19529196619987488, -0.9711461663246155, -1.0018576383590698, 0.28349006175994873, -0.7949268221855164, 2.7821292877197266, -1.1041209697723389, -0.10834793001413345, 0.027560703456401825, -0.0492086224257946, 0.7511670589447021, -0.5562565326690674, 0.28615474700927734, -1.3708993196487427, -1.2357267141342163, 0.9789676070213318, -0.8194782733917236, -0.7905557155609131, 0.4557397663593292, -1.5703837871551514, -2.315229654312134, 0.974853515625, 0.002639365615323186, -0.7340263724327087, 0.35426217317581177, 0.8329362869262695, -1.1457265615463257, -0.28049436211586, -0.10502022504806519, -0.8055132031440735, -0.18969053030014038, -0.895236074924469, 1.4006978273391724, -0.1492171734571457, 0.33577409386634827, -0.37628665566444397, -1.0993423461914062, 1.2455503940582275, 0.8640683889389038, -1.7979811429977417, 0.5054751038551331, -1.445978045463562, -0.22590272128582, -0.5644998550415039, -0.3276696801185608, -2.041170835494995, 0.60556960105896, -1.3243367671966553, -0.32066869735717773, -1.7829314470291138, -0.791534423828125, -0.3128912150859833, -0.5209714770317078, -0.1668739765882492, -0.8806526064872742, -1.635906457901001, 0.39586958289146423, 0.6579381823539734, -0.5848343372344971, 1.7462877035140991, 1.1500163078308105, -0.30959174036979675, -0.34487444162368774, 1.492688536643982, -0.19986781477928162, 0.8013609647750854, -0.6156800985336304, 0.7742857933044434, 1.2046223878860474, -1.3033815622329712, -0.037804488092660904, 1.0985469818115234, -2.163591146469116, 1.6905971765518188, 0.2654576003551483, 1.6545721292495728, 2.2184665203094482, -0.572970986366272, -0.40840694308280945, -0.07732567191123962, -1.2024400234222412, 0.8826473355293274, 0.20361296832561493, -0.32005277276039124, -0.3974684178829193, 0.10537488758563995, 1.4744919538497925, -0.23191511631011963, -1.6228537559509277, 1.9560033082962036, 1.0355664491653442, -0.8861817121505737, -0.47804754972457886, 0.7691356539726257, 0.07862914353609085, -0.10179781168699265, 0.6798424124717712, 0.9374602437019348, -0.7336583733558655, 0.8378309011459351, 0.16763801872730255, 0.1648576557636261, 0.09289254248142242, -2.8203461170196533, -1.1136889457702637, -1.0183768272399902, 0.024369535967707634, -1.7809191942214966, 1.151571273803711, -0.31550168991088867, -0.3330656588077545, 1.0527400970458984, 0.5120601058006287, -2.921546459197998, 0.4329930543899536, 1.4053127765655518, 0.624184787273407, 1.1041189432144165, 1.1228605508804321, -0.045451659709215164, 0.3458648920059204, -0.8275808095932007, -0.27669432759284973, -1.5932623147964478, 0.12039900571107864, 0.9550290703773499, -1.215882420539856, 0.08948823064565659, -0.2992914915084839, -0.6579218506813049], [-1.5439674854278564, 1.5551702976226807, 0.5393931865692139, 0.23822149634361267, 0.48130732774734497, 0.40116891264915466, 0.9625930190086365, -0.8150677680969238, -1.0251520872116089, 0.6027362942695618, 0.663293719291687, -1.8666701316833496, 0.8379412293434143, -0.214009091258049, -0.5307179093360901, 0.45261651277542114, -0.011394518427550793, -0.27789103984832764, -1.968375325202942, 0.09134486317634583, -0.8659397959709167, 0.7564696073532104, -0.8648778200149536, -0.9834994673728943, 0.2476840615272522, 1.3453993797302246, -1.3684732913970947, 1.4195884466171265, 0.330733060836792, 1.4341875314712524, 0.0141768679022789, 0.2303340584039688, 1.2755522727966309, -0.05897574499249458, -0.04249388724565506, 0.3189505338668823, -1.7369378805160522, -0.6809990406036377, 0.33709320425987244, -0.4661330282688141, 1.3227102756500244, 0.5974579453468323, -0.292955219745636, -1.1744273900985718, 1.0623141527175903, 0.6052709221839905, 0.7543134093284607, 0.3168339133262634, 1.8625528812408447, -1.1117446422576904, 0.9379337430000305, 0.09748496860265732, 0.24477939307689667, 0.06850186735391617, 2.1063668727874756, -1.1208356618881226, -0.7012729644775391, 0.06395591050386429, -1.472638487815857, -0.2517595887184143, 0.02727472595870495, -0.3241725265979767, 1.4996072053909302, 1.1430059671401978, -0.09366734325885773, -0.5310751795768738, -0.4652397036552429, -0.5811694264411926, -1.33382248878479, -0.5215263962745667, -0.4581717550754547, 0.7254959940910339, 0.36148545145988464, 0.8117417097091675, 0.4110763669013977, 0.7548640966415405, -1.9766901731491089, -0.7168071269989014, -1.7083158493041992, 0.4062541723251343, -0.24378687143325806, -0.5194084644317627, 0.10812294483184814, 2.543970823287964, -0.26355868577957153, 1.049080491065979, -1.1052217483520508, 0.015812357887625694, -0.7714837789535522, -0.3052871823310852, -1.8035327196121216, -0.4516682028770447, 0.9870926141738892, 2.014044761657715, 0.02967141941189766, -1.6083858013153076, 0.2595520317554474, 0.2782440185546875, 1.5293846130371094, -0.2320299744606018, 0.04988265037536621, -0.7874489426612854, 0.9349294304847717, -0.5675156116485596, 1.3318016529083252, -0.07715131342411041, 0.14377562701702118, -0.0742298886179924, -0.0856398418545723, -0.4738999605178833, -1.3213245868682861, -0.05811398848891258, 0.18531933426856995, -0.07415994256734848, -1.235964059829712, 1.1695505380630493, -0.22307905554771423, 1.2329065799713135, -0.6279005408287048, -0.5323209166526794, -0.5209933519363403, -2.3874480724334717, -1.3977367877960205, -0.2796921730041504, 0.7089199423789978, 0.8265318274497986, 1.7924706935882568, 1.8196427822113037, -1.374107003211975, -1.3888942003250122, 0.5561057925224304, 0.9399797320365906, 0.5080468058586121, -0.21380725502967834, 0.3054802119731903, 0.7384478449821472, -0.20377738773822784, 0.35203850269317627, -0.6458213925361633, 0.3781263530254364, -2.421370267868042, 0.4618191719055176, 0.21380026638507843, -0.7886303663253784, -1.3705934286117554, -0.6706259846687317, -0.978178858757019, -0.5369755029678345, -0.14929938316345215, -0.3441910147666931, -0.7762090563774109, 1.1382510662078857, -0.625634491443634, 0.16344356536865234, -0.08721465617418289, -0.9213693141937256, -0.2706346809864044, 0.1657067835330963, 0.4937605559825897, -1.1253447532653809, -0.7769258618354797, -0.39110010862350464, -1.3191417455673218, -0.4128355085849762, 1.3233188390731812, -1.2016407251358032, -0.4414224624633789, -0.925005316734314, -0.761680543422699, -1.7552183866500854, 0.10554789751768112, 1.1700360774993896, 0.023039428517222404, -0.2183624505996704, 1.3359456062316895, -0.33427780866622925, 0.34491974115371704, -0.7691372632980347, 1.1742521524429321, 0.15573078393936157, 0.8149869441986084, -1.4424874782562256, -1.6185202598571777, -1.12728750705719, 0.31121107935905457, 0.06027863919734955, -0.46303537487983704, 0.7316182851791382, -0.8867822885513306, -0.20429466664791107, -0.5974950194358826, 0.09329263865947723, -2.2066643238067627, -1.5735033750534058, -0.5545119047164917, -0.505354106426239, -0.6254482269287109, 0.00914502702653408, -1.1046040058135986, 0.8138746619224548, -0.6379592418670654, -0.05742643028497696, -0.043847374618053436, -0.1148146241903305, -0.3942984640598297, 1.8725454807281494, -0.637691855430603, -1.0698153972625732, 0.6310796141624451, 1.2213293313980103, 0.5794181227684021, 0.49892178177833557, -0.19579005241394043, 0.04474311321973801, -0.09142543375492096, -0.17269183695316315, -0.10901093482971191, -0.426420658826828, 0.11600430309772491, -1.71591317653656, 0.30683833360671997, -0.009490550495684147, 0.0064944555051624775, 0.034714676439762115, -0.060312677174806595, -1.1699670553207397, -1.461559534072876, -2.398000717163086, -1.419941782951355, -0.4886704683303833, -0.46484529972076416, -0.7377198934555054, 0.4967765510082245, 0.8790032863616943, -0.5075090527534485, -0.7640783786773682, 0.37805232405662537, -1.0075302124023438, -0.49640169739723206, 1.3053265810012817, 0.0777488648891449, 1.119500994682312, -0.7467970252037048, -0.289646714925766, -0.08786804974079132, -2.211970090866089, -0.9565623998641968, 0.39894363284111023, 1.2273566722869873, 0.5659080147743225, -0.9367823004722595, -0.42956027388572693, -1.172533392906189, -0.9266213774681091, -0.08349556475877762, 0.03452366217970848, 0.5288403630256653, -0.22660218179225922, -0.4733790159225464, 1.208272933959961, 0.00814240425825119, -0.07728936523199081, -1.417744517326355, -0.28294163942337036, -0.592253565788269, 0.5405322313308716, -0.047701653093099594, 1.0251222848892212, -2.1428372859954834, -2.2227323055267334, -1.5127092599868774, 0.1473446637392044, 0.7819164395332336, 0.010316666215658188, 0.7076069116592407, 0.6845212578773499, -0.2031671404838562, 3.3355894088745117, -0.016437940299510956, -0.3214380145072937, -0.1634596884250641, -0.5259292125701904, -1.155569076538086, -0.028488533571362495, -1.6059684753417969, 1.8131308555603027, 0.20747776329517365, -0.6777288317680359, -0.6068570613861084, 0.41528746485710144, -0.10490323603153229, 1.010048747062683, 0.8973783254623413, 0.09569886326789856, 0.39049312472343445, -0.9934417605400085, -2.2904131412506104, -0.00537894107401371, -0.2827078700065613, 1.719473958015442, -0.995885968208313, -0.0879405215382576, -0.3700927793979645, 1.5805633068084717, 1.2801313400268555, -0.6503173112869263, -0.4213004410266876, 0.8980705738067627, -1.3368918895721436, -1.0528008937835693, 1.089847445487976, -1.1954478025436401, -0.8005251288414001, -1.1170129776000977, -0.7514679431915283, 1.4777244329452515, 0.1622716188430786, -1.067136287689209, -0.8869413137435913, -0.7034611701965332, 0.1985974758863449, -1.0559933185577393, -0.11855518817901611, -0.7726444005966187, 0.4867667555809021, -1.4518237113952637, 1.7125523090362549, -0.8215951323509216, 1.7070976495742798, -1.5883653163909912, 0.4863279163837433, 0.7006510496139526, -0.16211622953414917, 0.4100920855998993, 1.7938542366027832, -0.5074625015258789, 0.2992790937423706, 0.004558540415018797, -0.2646704316139221, -0.05272642523050308, -0.3525258004665375, 1.2329275608062744, 1.4859576225280762, -2.4951446056365967, -0.9312376379966736, 1.0875051021575928, -0.7207589149475098, 1.3676787614822388, 0.029972970485687256, -0.5207818746566772, 2.5469279289245605, 1.353206992149353, 0.437913179397583, 0.18225549161434174, -0.8223322629928589, -1.264006495475769, 0.38817280530929565, -0.03801095858216286, 0.3175715506076813, 1.496332049369812, 0.5772692561149597, 0.10363089293241501, 0.36576494574546814, -0.28830084204673767, 0.5374362468719482, 0.02269473671913147, 1.6021939516067505, 1.6113786697387695, 0.2705284357070923, 0.5682139992713928, 0.5844910144805908, -2.2644999027252197, 1.47178053855896, 0.6207743287086487, -2.228766679763794, 0.9177126884460449, -0.044127244502305984, -1.0214076042175293, 1.0106810331344604, 2.616260051727295, 0.4284074902534485, 0.7698813676834106, -1.1418020725250244, 0.401395320892334, -0.5571988224983215, 0.8914965987205505, -0.7356139421463013, 0.09704003483057022, -0.8329465985298157, 0.5780982375144958, -0.8611320853233337, 0.05786114186048508, 0.9125260710716248, -0.11224204301834106, 0.0611727200448513, 1.2213581800460815, 0.27662360668182373, 0.05269848182797432, -1.0854477882385254, 0.5136814117431641, 1.1040008068084717, 0.19399267435073853, -0.04019112139940262, -0.08587048947811127, -0.2743378281593323, 2.1869513988494873, 0.5483234524726868, -1.0421124696731567, -0.7310324311256409, 1.0820666551589966, 1.0615580081939697, -0.348279744386673, 0.0734722763299942, -1.4504414796829224, 1.1321004629135132, -0.017680680379271507, 0.08884062618017197, 0.23742079734802246, -0.4440310001373291, 0.11421816051006317, 0.3198438882827759, 1.4383443593978882, -1.281148076057434, 0.5524168014526367, 0.4871751368045807, -0.15237918496131897, -0.3669153153896332, 0.14325466752052307, -0.25541436672210693, -1.5939279794692993, -0.5636976957321167, 0.1410413682460785, -0.6214174628257751, 1.797523021697998, 1.0646981000900269, 0.1476016789674759, 1.4270788431167603, -1.234808325767517, -0.1772640347480774, 0.543428897857666, 1.143330454826355, 0.3511473536491394, -1.3933113813400269, -1.2006635665893555, -1.6451553106307983, -1.6954222917556763, -0.5786424279212952, 1.9422883987426758, 1.4804081916809082, 2.0608956813812256, 0.019548391923308372, -0.690447211265564, 0.12014850229024887, -2.2080838680267334, 0.6796796321868896, -1.1015347242355347, -1.0689841508865356, -0.9438077211380005, -1.2414555549621582, -0.8477551937103271, -0.4748055040836334, 1.3465988636016846, 1.5313771963119507, 1.0271296501159668, -0.7804629802703857, -0.6342120170593262, -0.1654650866985321, -0.13096801936626434, -0.806359052658081, 1.150858998298645, 1.0129114389419556, -0.1256358027458191, -1.327641248703003, -1.6661745309829712, -0.17289461195468903, -1.3956348896026611, 0.4465464651584625, -0.7668564319610596, 1.4249241352081299, -0.484446257352829, 2.0723824501037598, -0.23417334258556366, -1.1550863981246948, -0.5057056546211243, 0.8250175714492798, -0.08587496727705002, -2.0285394191741943, 1.08506441116333, -0.1719844937324524, 0.9448775053024292, -0.1689465492963791, 0.5144883990287781, 0.07754740118980408, -0.44267037510871887, 0.940219521522522, -1.0131624937057495, -1.1429197788238525, -0.3909885287284851, 0.14597946405410767, -0.08252313733100891, 0.4712632894515991, -0.42127275466918945, 0.2919614613056183, -0.15358000993728638, 1.2799937725067139, -2.2429096698760986, 1.1817870140075684, 0.04296617582440376, 1.6438184976577759, 0.6360896825790405, -1.6609985828399658, -0.27076271176338196, 0.45977073907852173, -0.6874244809150696, 0.3368009924888611, -2.24063777923584, -0.47609782218933105, -0.45532315969467163, -1.1223835945129395, -0.7426412105560303, 0.3718545734882355, -0.8240483999252319, 0.30699431896209717, 0.6579498052597046, -0.6746594905853271, -0.14033961296081543, -0.009551260620355606, -1.4044215679168701, -0.8366169929504395, 0.8146523833274841, 0.1746479868888855, 0.8288945555686951, 0.19091038405895233, 0.0689253956079483, -0.08245808631181717, 1.556764006614685, 1.4738980531692505, -0.954831600189209, 0.482399582862854, -1.505662202835083, -1.4655306339263916, -1.3475451469421387, 0.19632254540920258, -0.30429884791374207, -0.33038216829299927, -0.2546616494655609, 0.24638254940509796, 0.18596884608268738, -1.7591276168823242, 0.7203471064567566, -0.89216148853302, 1.027601718902588, 0.5839819312095642, -1.032435655593872, -0.19031140208244324, 0.08371501415967941, -0.4790627062320709, -1.0298378467559814, -1.31661057472229, 1.4501819610595703, 1.0928614139556885, 0.09701599925756454, 0.035950131714344025, -0.6866835355758667, -0.06528441607952118, 0.5114946961402893, -1.7661341428756714, 0.5035036206245422, 0.9744933247566223, 0.49947890639305115, -0.6651323437690735, 1.332306146621704, 0.5088133215904236, -0.4132523834705353, 0.08351796865463257, -0.5813781023025513, -1.252077341079712, -0.2247966080904007, -0.06364792585372925, 0.8782910108566284, 0.5039067268371582, 1.0948110818862915, -0.06648891419172287, -0.8036482334136963, -1.2132617235183716, -0.8259192705154419, -0.34350496530532837, 1.1080052852630615, 1.4829328060150146, -0.46830490231513977, 0.9710693955421448, -0.3317352831363678, -0.9786157608032227, 0.7174282073974609, -0.9328517317771912, 0.31345757842063904, -0.06264644116163254, -0.3725190758705139, 1.2753326892852783, -2.2764880657196045, 0.14385488629341125, 1.3532531261444092, 0.13493919372558594, 0.11027833074331284, 0.3279821574687958, 0.34588590264320374, 0.2571559250354767, -0.752513587474823, 0.24670694768428802, 0.6563573479652405, -1.314561367034912, -0.11665556579828262, 0.5447710752487183, 0.31503599882125854, -0.11852633208036423, 0.43936148285865784, 0.21336843073368073, 1.0757650136947632, -0.5151311159133911, 1.328262448310852, 0.2510976493358612, -1.462272047996521, 1.1740590333938599, -0.6327944397926331, -2.7459638118743896, 1.8660236597061157, 0.36722874641418457, 1.0296168327331543, 0.15804818272590637, 0.09609111398458481, -0.13152053952217102, -0.31791916489601135, 0.9271764755249023, -0.46499162912368774, 1.2595012187957764, 0.1991475522518158, -1.2291432619094849, -0.05235673114657402, 0.518699049949646, 0.5138448476791382, -0.512263298034668, 0.9875613451004028, 0.4685341715812683, 1.7386168241500854, -0.8995159268379211, -0.059211671352386475, -1.303685188293457, 0.32045406103134155, 0.8433080315589905, 0.4056447148323059, -0.4094296097755432, 0.7513188719749451, 0.9826366305351257, -0.04284687712788582, 0.8655581474304199, -0.47137463092803955, -0.34999096393585205, -0.9295687675476074, -1.183820366859436, 0.63862544298172, -0.17068015038967133, -1.0689340829849243, -0.33130842447280884, 0.030350226908922195, -2.0143215656280518, 0.1092735230922699, 0.22040027379989624, 0.7121303081512451, -0.8908945322036743, -1.1386456489562988, -1.1236038208007812, 0.4966071546077728, -1.7920536994934082, 0.35491999983787537, 1.2895994186401367, -0.1101473793387413, 0.5246129631996155, -0.0362871028482914, 0.32311102747917175, -1.5045455694198608, -0.43016695976257324, 0.9149773716926575, -0.8130759596824646, -0.9992561936378479, -1.1874357461929321, -0.9990484714508057, -0.34102359414100647, -1.3106906414031982, -1.7399725914001465, -0.7767189145088196, -2.031484842300415, -1.2373305559158325, -0.2552550733089447, 0.34722280502319336, 0.45595836639404297, 0.0010978598147630692, 0.7245436906814575, -0.7802380323410034, -0.9196276664733887, -4.7741875648498535, 0.7128912806510925, -0.9961326122283936, -0.8078412413597107, 0.6258613467216492, -1.3300832509994507, -0.8390744924545288, -0.2541590929031372, 0.3259696662425995, -0.5614232420921326, 0.7961768507957458, -0.786271870136261, 1.0434563159942627, 0.1575428545475006, 1.4149216413497925, 0.2909967303276062, 0.9832534193992615, 0.5875442624092102, -1.9569432735443115, 1.1329536437988281, 0.17994341254234314, 0.017467191442847252, -0.7949745655059814, 0.5582972764968872, -0.8730318546295166, -0.0317668542265892, -1.2443073987960815, -0.18383850157260895, 1.1026266813278198, -0.43199530243873596, 0.38089823722839355, -0.3755556046962738, -0.3045559227466583, 1.5789324045181274, 1.2287211418151855, -0.07359106093645096, 0.2142443209886551, 1.8440015316009521, -0.3113183379173279, -0.18530941009521484, -0.7978028059005737, -1.2029643058776855, -0.5596740245819092, 0.33505284786224365, -1.1423118114471436, -0.05326998233795166, -0.19504396617412567, -1.0226719379425049, -0.662358283996582, -0.6054167747497559, 0.47130295634269714, -0.6181070804595947, 1.5010069608688354, 0.6388646960258484, -0.30430305004119873, 1.1601159572601318, -0.4223381578922272, 0.7286860942840576, 0.5630804300308228, -0.7505962252616882, 2.860616445541382, 0.2514037787914276, 0.6387149691581726, 0.29615500569343567, 1.1450278759002686, -1.3053423166275024, 0.7141181826591492, 3.144648790359497, 0.8866302371025085, -0.03508397191762924, 0.30774492025375366, 1.2936058044433594, -0.6322095990180969, 0.4174671471118927, 0.6564743518829346, -0.18380701541900635, 0.14200150966644287, -0.14016474783420563, 0.8476378321647644, 0.8546160459518433, 0.321037620306015, 0.15678486227989197, 1.2156065702438354, -0.4048497974872589, -1.6370748281478882, -0.2925311326980591, 0.6035144329071045, -0.5832739472389221, -0.41235631704330444, 0.9773173332214355, -0.5644913911819458, 1.389471411705017, 0.7435264587402344, -0.340078741312027, -1.201195240020752, 1.1903306245803833, -0.5440386533737183, 2.467888355255127, -1.1954842805862427, -1.107688069343567, 0.977003276348114, 0.11653002351522446, 1.4379463195800781, -0.0958266407251358, 1.2914220094680786, 0.35107821226119995, -0.7693018913269043, -0.8906651139259338, -0.36760178208351135, -0.35846757888793945, -0.393120139837265, 1.2310115098953247, -1.7192370891571045, 0.06128925085067749, 1.1949316263198853, 0.6034736037254333, -1.2557207345962524, -1.6412327289581299, 0.6776511073112488, -0.7195844054222107, 1.121634840965271, -0.23055382072925568, 0.39529651403427124, 2.3643174171447754, -0.428803026676178, -1.035194993019104, 0.5656346082687378, -1.7079063653945923, 0.7898312211036682, -0.380784809589386, 0.42691704630851746, -0.8057702779769897, -1.2490452527999878, -0.25628790259361267, -0.9166399240493774, 1.1060794591903687, 0.9037191271781921, -1.1724226474761963, 1.2997677326202393, 2.6152706146240234, 2.407169818878174, -0.3625411093235016, -0.33373913168907166, -0.04716271162033081, -1.06254243850708, 0.9929345846176147, -0.3248654901981354, -0.2422156184911728, -0.8086626529693604, -0.709317684173584, -0.581281840801239, -2.2771148681640625, -1.6746183633804321, 1.7039976119995117, -0.7364645004272461, -0.4617476761341095, 1.0432003736495972, -0.87796550989151, -0.22735165059566498, 0.746367871761322, -0.5803439617156982, -1.3952683210372925, -0.6130475997924805, 1.3079618215560913, -0.8655604720115662, -1.0466386079788208, -1.0681685209274292, 0.44812852144241333, -1.4919148683547974, -0.2128230333328247, 0.5196502208709717, -0.5613994598388672, 0.5023988485336304, 1.0383824110031128, 1.489911437034607, 0.034848880022764206, -1.3615798950195312, 0.42050492763519287, -0.4288598299026489, -1.3115360736846924, -0.37443697452545166, 0.909254252910614, 1.030365228652954, 1.4773077964782715, 1.4972906112670898, -0.07452621310949326, 0.2652674615383148, 0.04477173835039139, 0.4057287573814392, 0.34877026081085205, 0.5813742280006409, -1.7322733402252197, 0.8308625817298889, -1.4211965799331665, -0.22706376016139984, 0.27457961440086365, 0.43242961168289185, 0.1454031616449356, -0.853828489780426, 0.8544243574142456, -0.44102761149406433, -1.402022361755371, 1.8082630634307861, -1.5633866786956787, -0.9166461229324341, 1.4549070596694946, 1.8103232383728027, 1.1466013193130493, -1.8984222412109375, 0.6872605085372925, -0.7919642329216003, -1.2565678358078003, 0.9147840738296509, -0.22048960626125336, 0.6041650176048279, -0.04525038227438927, 0.9316869974136353, 2.403394937515259, -0.9822475910186768, -0.49109703302383423, 2.5175085067749023, 0.24915297329425812, -0.5291211009025574, -1.2236230373382568, 1.1300947666168213, -0.34314587712287903, -0.1598186194896698, -0.32021811604499817, 0.5527113080024719, 1.2278739213943481, 1.161511778831482, -1.3739012479782104, -0.35371923446655273, 0.6742503643035889, -1.502148151397705, -0.8097607493400574, 0.11547344923019409, 0.12416859716176987, -1.1435641050338745, -1.8165473937988281, 0.4360957145690918, -2.9013898372650146, 0.6215922236442566, -1.5581117868423462, 1.4032479524612427, 0.7573703527450562, 1.327085018157959, 0.45556801557540894, 1.6526061296463013, 0.16318552196025848, -0.16451209783554077, -1.5064728260040283, 0.06126237288117409, -0.41555073857307434, 1.6010812520980835, 0.13423201441764832, 0.5053165555000305, -0.13656122982501984, 0.1498592048883438, -0.23440609872341156, 1.7333647012710571, 0.7303178310394287, -0.9522871375083923, -0.8158779740333557, 0.9997672438621521, -0.38366222381591797, 0.4444624185562134, -0.48667091131210327, 0.0861290767788887, 0.3577014207839966, -1.4885883331298828, -1.695466160774231, -1.2154041528701782, 0.6198990941047668, 0.1347954273223877, 1.96195650100708, -0.3581766188144684, 0.6042655110359192, -0.6599398255348206, 0.4681969881057739, -0.03267653286457062, -0.3703121244907379, 3.046579599380493, -0.08055765926837921, -0.5600824952125549, 1.1992510557174683, -2.0185797214508057, 0.8197519779205322, 1.2006397247314453, 0.020759273320436478, -0.5876589417457581, -0.42424002289772034, 0.6929650902748108, -1.4207793474197388, 1.1777018308639526, 1.1948485374450684, 0.13352929055690765, 0.23150698840618134, -0.4974236786365509, -0.9319105744361877, 0.7730453014373779, 0.8192901015281677, 2.6836891174316406, 2.4865052700042725, 1.1291167736053467, 1.5611671209335327, -1.2799010276794434, -0.612091600894928, 0.17204497754573822, 0.3783845007419586, -1.8921245336532593, 1.3336846828460693, 0.22463257610797882, 1.4341284036636353, 2.069254159927368, 0.5472803115844727, 0.41572436690330505, -0.2823482155799866, -0.4976714849472046, 0.5757638812065125, 0.8730629682540894, 0.725176215171814, 1.3950989246368408, 0.21970424056053162, -2.726257085800171], [-0.5345560312271118, 0.8109943270683289, 1.1906081438064575, -0.9782529473304749, 0.27939489483833313, -0.8012962937355042, 0.14759831130504608, -0.25527462363243103, 0.9575636982917786, 1.0760726928710938, -0.5712118148803711, -0.14797554910182953, -1.5027762651443481, -0.5211567878723145, 2.1548805236816406, -1.3282428979873657, 0.27680516242980957, -0.3446718454360962, 0.17300033569335938, -0.36138907074928284, 0.8858752250671387, 0.4462774395942688, -0.42544984817504883, 0.16882629692554474, 1.8742477893829346, 0.7721284031867981, 0.18637214601039886, -0.5368148684501648, 2.2691211700439453, 1.1722804307937622, -0.18416695296764374, -2.475637912750244, -0.09889216721057892, -1.7299628257751465, 0.2835548222064972, -0.5543180704116821, 0.17797262966632843, 0.638542890548706, 0.2316148430109024, 0.26646214723587036, 0.8305751085281372, -0.40549132227897644, 0.22159257531166077, 0.4734109342098236, 1.5526659488677979, -0.9202156066894531, 2.0204007625579834, -0.7435446381568909, 0.5302842259407043, -1.5203324556350708, 1.2230042219161987, -0.5320037007331848, 0.7325708866119385, -0.3667050898075104, -0.6838494539260864, 0.42553749680519104, -1.002321481704712, 0.5061331391334534, 0.03780916705727577, 1.3308342695236206, 0.7947143912315369, -0.013432535342872143, 0.9311456084251404, -0.6039121747016907, 1.2947255373001099, 0.17639338970184326, -0.8959193825721741, -0.876028835773468, -0.01650574430823326, 1.3018252849578857, -0.3705138564109802, 0.6153842210769653, 0.2767738699913025, 0.4658568203449249, 2.2488439083099365, 0.6278321146965027, -0.3394331932067871, 0.0908748209476471, 0.46394315361976624, -1.9696204662322998, 5.549942216021009e-05, 0.6584832668304443, -0.2164141833782196, -1.097667932510376, -0.2861379086971283, -0.2991923689842224, -0.6312736868858337, -0.6187363266944885, 0.5105153918266296, 0.7839170098304749, -0.5308132767677307, -0.8949755430221558, 0.9530941843986511, -0.7925945520401001, -0.5523958802223206, -0.18046198785305023, 0.24741630256175995, -1.2214535474777222, 1.1498554944992065, 2.002429485321045, 1.0057851076126099, 0.07493741065263748, -0.7846850752830505, 1.2299853563308716, -0.04200288653373718, 0.5205599069595337, -0.8596537709236145, -1.3236287832260132, -0.22078721225261688, 0.06550756841897964, -0.6983886957168579, -0.6138124465942383, 0.6468594074249268, -1.3381625413894653, 0.07383614778518677, -1.0973018407821655, -0.763326108455658, 0.35883715748786926, -0.31933778524398804, 1.8492802381515503, 0.00680821668356657, -0.7347911596298218, -0.7078380584716797, -2.5445263385772705, -1.8000411987304688, -0.47856947779655457, 0.13489274680614471, -0.16268596053123474, -0.2792799770832062, -0.5380964279174805, -0.13246692717075348, -0.31691810488700867, -1.1680874824523926, -0.8607782125473022, 0.5334065556526184, 0.20512250065803528, 0.5723755359649658, -0.18865224719047546, -0.3519686758518219, 0.5046270489692688, -0.31689226627349854, 0.07096106559038162, -0.413088858127594, 1.250873327255249, 0.9465317130088806, -0.5196064710617065, 1.2538464069366455, -0.05315554887056351, -1.6101312637329102, -1.0343002080917358, -0.4752906560897827, 0.559905469417572, -0.5792722702026367, -0.008544673211872578, -0.28431394696235657, -1.2175018787384033, -0.475515753030777, -0.7840022444725037, 1.3269635438919067, -0.18132250010967255, 2.1659412384033203, -0.786462128162384, -1.0236523151397705, 0.39463120698928833, -0.6162062883377075, 0.710428774356842, -1.9403080940246582, -0.4266522526741028, 0.6886640191078186, -1.0291296243667603, -0.8099507093429565, -2.022186279296875, -0.4514814019203186, 0.24576333165168762, 0.06792324781417847, -1.3436366319656372, -0.09650424867868423, 0.5423882603645325, 0.25085437297821045, 1.681951642036438, -0.412911593914032, -1.0197821855545044, -0.4624955356121063, -1.2321659326553345, 0.03536926209926605, -1.337135672569275, 0.08064286410808563, -1.513443946838379, -0.5766123533248901, -0.3570685386657715, 1.6213889122009277, 2.5348145961761475, -0.28306230902671814, -1.1791497468948364, 0.7648196220397949, 0.9097468256950378, -1.496563196182251, 0.3988989293575287, -0.4072808623313904, 1.0655931234359741, -1.573100209236145, 1.5311254262924194, 0.8004472255706787, -0.3866216838359833, 1.2723201513290405, -0.749621570110321, 0.05865633115172386, 0.3055834472179413, 1.4283021688461304, -1.2854684591293335, -0.3253849744796753, -1.4133424758911133, -0.7986273765563965, -1.344302773475647, -0.3312760889530182, 2.2302870750427246, 1.5477733612060547, 0.09990296512842178, 0.06750967353582382, -0.1325221210718155, -0.8384325504302979, 0.7961011528968811, -0.0020105375442653894, 0.22720228135585785, -0.9785286784172058, 2.8391356468200684, -0.5987325310707092, -1.0610674619674683, -0.028104273602366447, -0.5556963682174683, -0.4690224826335907, 0.1824028491973877, -0.7237110137939453, 1.3871041536331177, -1.3995919227600098, 2.3232922554016113, 0.08855076879262924, 1.1575862169265747, 1.5282676219940186, 0.8441879749298096, -0.9643967747688293, -0.29860344529151917, 1.3802573680877686, -0.7062469124794006, -1.0215027332305908, -1.4755924940109253, -1.8313615322113037, -2.9020912647247314, -0.1067621037364006, -0.33521926403045654, 0.8727968335151672, 0.0010926991235464811, 1.3999738693237305, 0.12532997131347656, 0.25244659185409546, -1.324637532234192, -1.2038187980651855, 1.2895835638046265, -1.0627732276916504, -0.15079990029335022, -0.5515325665473938, 1.8194913864135742, -0.4840501546859741, -0.9724728465080261, 1.204616665840149, 1.4445571899414062, -0.9243437051773071, -0.6906586289405823, 0.6181759238243103, -0.5897256731987, -0.6082763671875, -0.16612489521503448, -0.21618111431598663, 0.09460414946079254, -1.9754773378372192, -0.1767418384552002, 0.08135208487510681, -0.11222457885742188, -0.14896148443222046, -1.3639837503433228, -0.06451553851366043, -0.490398108959198, 1.709585428237915, -0.6647516489028931, 0.47224289178848267, 1.0015605688095093, -1.6551915407180786, 0.13081641495227814, 0.1318127065896988, -0.5196698307991028, 1.1156319379806519, -0.9505385160446167, -1.1730972528457642, 0.28097400069236755, -0.26369571685791016, -1.0480209589004517, 0.985449492931366, -1.4465746879577637, -0.27325955033302307, -0.8693058490753174, 0.8780749440193176, 2.4850471019744873, 0.4667866826057434, -0.6274306178092957, 1.2940059900283813, 0.03274436667561531, -0.12829451262950897, -0.17035920917987823, -0.32009658217430115, 0.10099271684885025, -1.3468974828720093, -0.9820697903633118, -0.11530985683202744, 0.2378341406583786, -0.6847776770591736, -0.295043408870697, 0.487460732460022, -0.05193088576197624, -0.6179537177085876, 1.1136837005615234, 0.10100854188203812, -1.6178572177886963, 0.7269332408905029, -0.7093784809112549, 0.08831685781478882, -0.5296170115470886, -0.15972426533699036, -1.6748669147491455, 1.0172734260559082, 0.6146994829177856, 0.5140112638473511, -1.202812910079956, 0.592775821685791, -0.3468177318572998, 0.720196008682251, 0.2739807367324829, 0.3265436887741089, -0.4463330805301666, -0.12933364510536194, -2.45123553276062, -1.8735079765319824, 0.6648755669593811, -0.8085551857948303, 1.6596161127090454, 0.5470333695411682, -1.298412799835205, 0.8036299347877502, 0.001968043390661478, -0.2818097472190857, 0.46668368577957153, -0.25545087456703186, 1.8859084844589233, -0.6953573226928711, 0.14442913234233856, 0.5797569751739502, -1.1632298231124878, 0.39132389426231384, 0.018533335998654366, 0.9870660901069641, -2.1551520824432373, 0.4495265483856201, 1.737589716911316, 0.44661155343055725, -0.4950416386127472, -0.9777652025222778, 0.7211661338806152, -0.28835421800613403, 0.37921810150146484, 0.6238473057746887, -0.8969466090202332, 0.21532969176769257, 0.7842593789100647, -0.03554343059659004, 1.6472680568695068, -1.9896926879882812, 0.9395238161087036, -0.16876204311847687, 1.5629485845565796, -0.611263632774353, -1.3406236171722412, 0.14518168568611145, -1.304088830947876, 1.6779507398605347, -0.13985994458198547, -0.8831449151039124, -1.985207438468933, 0.3933369815349579, 0.2997342646121979, 0.698677659034729, -1.4879189729690552, 0.4541252553462982, -0.016394589096307755, 1.7361936569213867, -0.5382463335990906, 1.1801786422729492, 0.9361090660095215, -0.9210319519042969, -1.3461871147155762, -0.6024870872497559, 1.5511445999145508, -0.7968538999557495, -0.8563730716705322, -1.0997116565704346, 1.1493098735809326, 0.03236938267946243, -0.9211146235466003, -0.4987965226173401, 0.27095532417297363, -0.5899145007133484, -0.4993126392364502, -0.628416895866394, 1.3019698858261108, 0.8760904669761658, 1.2646950483322144, -1.5846962928771973, 1.9574440717697144, -0.7340303659439087, -0.2387201488018036, 0.5110026597976685, 1.0894538164138794, 0.22922629117965698, 0.7582825422286987, 0.18224753439426422, -1.6397390365600586, -1.617414116859436, -0.34895893931388855, 0.1788952797651291, 0.11219726502895355, 1.2653685808181763, 0.8010408282279968, 0.4243280291557312, 0.48415324091911316, 2.261375904083252, 0.14329224824905396, -1.236124873161316, -0.8942975997924805, 1.259338140487671, -0.3509048521518707, -0.8408817052841187, 0.2873391807079315, 0.039806950837373734, -0.28434598445892334, 0.7273544073104858, -0.7092165946960449, -1.8062373399734497, 1.2882630825042725, -0.10868936032056808, 0.493723601102829, 1.3402471542358398, -0.604408860206604, 0.42497026920318604, 0.4924183487892151, -0.6321818232536316, 0.11650457978248596, 0.5239144563674927, -1.4542386531829834, 0.1324087679386139, -0.423129141330719, 1.043519377708435, -1.0136754512786865, 1.2556774616241455, -0.6241099238395691, 1.952977180480957, -0.49308764934539795, 1.909519910812378, -1.4161195755004883, -0.5218896865844727, -2.506946563720703, -1.1992501020431519, 1.328790545463562, -0.7813396453857422, -0.10263931751251221, 0.8569849133491516, 0.5946117043495178, -0.013684386387467384, 0.16785459220409393, 0.9016928672790527, 1.6224219799041748, -0.4108082056045532, -0.027348270639777184, -0.6830599904060364, -0.7569432854652405, 0.18766726553440094, 1.7943735122680664, -1.2896394729614258, -0.10033974796533585, -0.9004691243171692, -1.0993516445159912, 1.3998557329177856, -1.1132608652114868, -0.8966084122657776, 0.3382676839828491, -0.44465628266334534, -0.8809065818786621, -0.20968155562877655, 0.09024441242218018, -0.1935107558965683, -0.017491549253463745, 0.2646988034248352, -1.9231793880462646, -0.3380625545978546, 0.8592895865440369, -0.8910073637962341, 0.11858834326267242, -1.3167815208435059, -0.8885373473167419, 2.164938449859619, 0.795505166053772, 1.2851920127868652, -0.3565332293510437, 0.993562638759613, 0.3858899176120758, -0.3494235873222351, -0.32403287291526794, 1.2441701889038086, 0.7699899077415466, 0.5916225910186768, -0.4462929666042328, -0.9241219162940979, -0.6201262474060059, -1.4434739351272583, -0.9963794350624084, 0.8897755146026611, -1.746997594833374, -0.448711097240448, -1.952709436416626, 2.6420414447784424, -0.507281482219696, 0.8774654269218445, -0.6795042753219604, -1.0527725219726562, -1.7702757120132446, -0.9477060437202454, -0.6517709493637085, -2.356790781021118, 0.05241326242685318, 1.0614110231399536, 0.34609711170196533, 0.8895903825759888, -0.8756392598152161, -0.45317134261131287, -1.088956594467163, 0.4151819944381714, -1.3657439947128296, -0.11897343397140503, 0.7015277743339539, -1.2646944522857666, 0.8695270419120789, 1.5973289012908936, -0.424826055765152, 0.03400152176618576, -0.5968995690345764, 1.0124260187149048, 0.5815103054046631, -1.2428689002990723, 0.2891749441623688, 0.14369513094425201, -0.22249643504619598, -0.3198080360889435, 0.5240107178688049, -0.5777169466018677, -1.839110016822815, -1.0270097255706787, -1.0572283267974854, 0.5114331841468811, -0.8297224044799805, -0.4312494993209839, 0.22005827724933624, 0.22429095208644867, 0.6061785221099854, 0.4821680188179016, 0.04595901444554329, 0.7189704179763794, -1.7590166330337524, 0.5561745762825012, 0.4106670022010803, -1.737616777420044, -1.2720812559127808, -0.16296206414699554, -1.051337718963623, 0.8725348711013794, -1.4001935720443726, -0.3426821231842041, -1.0837299823760986, 0.9048617482185364, -0.6968464851379395, -0.28885480761528015, 1.9254051446914673, -0.054865527898073196, 0.8912988901138306, -0.5482256412506104, -0.6458135843276978, 0.3328060507774353, -0.4437641501426697, -0.45464009046554565, -2.4292972087860107, -1.8963502645492554, 0.5875203013420105, 0.18574537336826324, -0.44706520438194275, -1.0340172052383423, -2.208073854446411, -0.01565847545862198, 0.35478878021240234, -1.141143798828125, -2.4479992389678955, -0.6631224155426025, 0.3633166551589966, -0.8847958445549011, 0.361803263425827, -2.083482027053833, 0.35853251814842224, -0.21294310688972473, -0.10047361999750137, 0.03928983956575394, 0.3045916259288788, 0.9971345067024231, 1.662061095237732, -0.711100697517395, -0.14917291700839996, -0.8238441348075867, -0.1462109237909317, -1.3061408996582031, 2.221653938293457, -0.992758572101593, 2.266873598098755, 1.0135339498519897, 0.5364441275596619, -0.9644085168838501, 0.3953651487827301, 1.5399250984191895, 1.1792579889297485, -1.0749375820159912, 0.30252546072006226, 0.5125749707221985, 0.02933168038725853, 0.8685528039932251, 0.3842616677284241, -1.1323895454406738, 0.33102214336395264, -0.7464321851730347, 0.9178506731987, -0.712858259677887, 0.33455607295036316, 0.6681368947029114, 1.0518633127212524, -0.4290124177932739, -0.14601866900920868, -0.07174908369779587, -0.4245404601097107, 0.06485608220100403, 2.069889545440674, 0.15544041991233826, 0.07205027341842651, 0.6584652066230774, 0.9270949959754944, 0.05557270348072052, -1.115595817565918, 0.9388976097106934, -0.13090690970420837, -0.5986192226409912, -0.8297860622406006, -1.0592719316482544, -0.8179640173912048, 0.6353917717933655, -0.16935651004314423, 0.9106709361076355, 1.8634892702102661, 0.37483862042427063, -0.7599073648452759, 0.8622190356254578, -1.804895043373108, 0.20879483222961426, 0.08614877611398697, -0.014127534814178944, -1.0771235227584839, -0.12592913210391998, 0.003309105057269335, 1.04681396484375, -1.0272152423858643, -1.0195808410644531, -1.944706916809082, -1.234939694404602, -0.5027405023574829, 0.7603111863136292, -1.3106008768081665, -0.2139836996793747, -0.6180418133735657, -1.3263143301010132, 0.3990935683250427, -0.629507303237915, 1.8510560989379883, 0.6203440427780151, -1.5344783067703247, -1.4291448593139648, 0.0007304793689399958, 1.7077438831329346, -0.36894330382347107, 2.2604806423187256, 0.400831937789917, 1.5396138429641724, 1.5873162746429443, -1.2289959192276, -0.7618326544761658, -0.6966603994369507, -1.251600742340088, -1.115775227546692, -1.1219300031661987, 1.3701512813568115, -0.1236129105091095, -0.08034069091081619, -0.8416286110877991, -0.14790193736553192, 1.1106016635894775, -0.06704016774892807, -1.3302079439163208, 1.2071775197982788, -0.6909304261207581, -0.7179268598556519, 1.2337846755981445, -0.26131150126457214, 0.9597049355506897, -0.45362162590026855, -0.16058972477912903, 1.8427207469940186, 0.3247690200805664, -0.8048546314239502, -1.8775250911712646, 1.189385175704956, 1.5038982629776, 0.0239779781550169, -2.0035400390625, 0.6647741794586182, -0.8070761561393738, 0.7482160925865173, -2.440253734588623, -0.040209896862506866, 1.1215245723724365, 0.18467704951763153, -1.324171781539917, 0.6488012671470642, 0.6565594673156738, 1.1653048992156982, 0.35234472155570984, -0.0019450259860605001, -0.07156805694103241, -0.7795520424842834, 0.6591708064079285, 0.965042233467102, 0.7621703147888184, 1.435240387916565, -0.862891674041748, -1.476623773574829, -1.0470473766326904, 1.1540120840072632, 0.9615974426269531, -1.7079497575759888, -0.24543602764606476, 0.08746463805437088, -0.6385072469711304, 0.41261887550354004, 0.09975749999284744, -0.13554231822490692, 0.6332013010978699, -0.6954503655433655, -0.4046836793422699, -1.8334742784500122, -1.9763339757919312, -1.4487717151641846, 0.05134838819503784, -0.6541407704353333, 0.4313277304172516, -1.4381251335144043, 0.5673360824584961, -1.2600802183151245, -0.5433146357536316, 1.7936784029006958, -0.7763946652412415, -1.2671438455581665, -0.8254117369651794, 0.03550029918551445, -0.663266658782959, 1.7662920951843262, 0.062278371304273605, 0.0016498935874551535, -1.0348559617996216, -1.513546109199524, -0.7823823094367981, -0.6242089867591858, 0.7405429482460022, 0.07536354660987854, 0.30264657735824585, -0.17439809441566467, -0.1425187736749649, -0.7150074243545532, -0.6812067031860352, 0.011774171143770218, -1.131162405014038, 1.4802707433700562, -0.9753835797309875, 0.998434841632843, -1.390798807144165, 1.6682085990905762, 0.7135125994682312, -0.7485993504524231, -1.4418331384658813, -0.5460863709449768, -0.08410357683897018, 0.29125919938087463, 1.328766942024231, 1.8773880004882812, -0.10595888644456863, -1.6442159414291382, 1.7684664726257324, -0.5566613078117371, -1.3244141340255737, 0.6318435668945312, -0.49572131037712097, 0.9722489714622498, 0.3893066942691803, -1.2480756044387817, 1.1700867414474487, 1.9945006370544434, 0.4747254550457001, 1.5991603136062622, 0.14222334325313568, -0.29027190804481506, 0.4089217782020569, -0.8118249773979187, 1.5637238025665283, -0.6202985048294067, 0.46454665064811707, -0.4416339099407196, -1.2274047136306763, -0.3223245441913605, 0.05685671046376228, 0.22504040598869324, 0.26310232281684875, 0.12745516002178192, -0.8684550523757935, -0.8464133143424988, -1.017033576965332, -0.592605710029602, -1.0907583236694336, -0.5118255615234375, 0.6138266921043396, -0.5584812164306641, 0.8024380803108215, 1.0257731676101685, -0.9620938301086426, -0.20587493479251862, -0.6228926777839661, 0.7330136895179749, -0.24371777474880219, 1.385224461555481, -0.06920040398836136, 0.2695549428462982, -0.13113680481910706, -0.6945700645446777, -0.3605302572250366, -0.8909628987312317, -0.8043432831764221, -0.7594051957130432, -0.9964126944541931, -0.4935581088066101, 0.1863163262605667, 0.04792044684290886, -1.162803292274475, 0.22925333678722382, -1.3618332147598267, 0.8582234978675842, -0.34492480754852295, -1.1004555225372314, -0.2736891210079193, 0.5609744191169739, 1.4816430807113647, 0.17786884307861328, -0.11005836725234985, 1.4141265153884888, -1.8020448684692383, 0.7849667072296143, -0.04002094268798828, -0.2505219280719757, -0.46413594484329224, 1.0759539604187012, -0.4824772775173187, -0.1331731528043747, -1.728947401046753, -0.898537278175354, 0.5179175138473511, 0.8609407544136047, 1.2261158227920532, 2.0873093605041504, 1.1048909425735474, -0.38644975423812866, -0.4219626486301422, -0.5261538028717041, 0.4954397976398468, 0.12505874037742615, 1.012819528579712, 0.0390460230410099, -0.5541816353797913, 0.906947135925293, -0.43771469593048096, -0.7603601813316345, 0.8755822777748108, -2.0345852375030518, 0.0009015556424856186, -0.020649053156375885, 0.8109456896781921, -1.068485140800476, 0.08121368288993835, -1.3505109548568726, -0.33448508381843567, 1.234911322593689, 1.2378966808319092, -0.5752348303794861, 0.48799532651901245, 1.5883172750473022, 0.16864921152591705, 0.8168467879295349, 0.031376879662275314, -1.1976313591003418, 0.4937622547149658, 0.8066965341567993, -0.8590542078018188, -0.7313345074653625, -0.1888071894645691, 0.24834899604320526, -0.0320163257420063, -1.0876178741455078, -0.09621801227331161, 0.6534016728401184, 0.5380606651306152, 0.16353626549243927, -0.11168999969959259, 2.1138947010040283, -1.1742627620697021, -0.7122644186019897, -0.39528703689575195, 0.08692031353712082, -2.684670925140381, -0.20927895605564117, 1.2091244459152222, 0.4094153046607971, -0.8438935875892639, -0.32154789566993713, 1.6614058017730713, -0.491699755191803, 0.06750679761171341, 2.76753830909729, -1.646640419960022, 2.0149667263031006, 0.6777418851852417, 0.14862923324108124, -0.06335011124610901, 1.3692100048065186, 0.6300651431083679, 0.7120679616928101, -0.8233393430709839, -1.0572665929794312, 0.7495948076248169, 0.3679594099521637, 2.3357391357421875, -0.4602597653865814, 0.11198945343494415, 0.7570312023162842, 0.25499090552330017, 1.0054782629013062, -2.48948073387146, -1.277320384979248, -0.8692611455917358, -0.6487919688224792, 0.913415789604187, 0.5532640218734741, -0.6343072652816772, 0.23863713443279266, -0.33409383893013, -1.3559986352920532, -1.2480500936508179, 0.9136031866073608, 1.3434627056121826, -2.1818478107452393, -1.6981139183044434, 0.6626375913619995, -0.13827620446681976, 1.3276265859603882, 0.251628577709198, -0.9643105268478394, -0.01358155719935894, 0.14005380868911743, 0.17886769771575928, 1.0763150453567505, 0.22131267189979553, 0.39955607056617737, 1.2514280080795288, 1.7328130006790161, -0.9470884203910828, 1.2000588178634644, 0.8625119924545288, -0.47569793462753296, 1.100996494293213, -0.6594087481498718, 0.6433730721473694, 3.0270090103149414, 2.2527048587799072, -0.3805662989616394, 0.7918658256530762, 1.2276661396026611, -0.1482994705438614, 2.2852814197540283, 0.12513913214206696, -1.0084569454193115, 0.2755773961544037, -0.020539920777082443, 0.458404004573822, 0.08055087178945541, 0.052748773247003555, 0.07270405441522598, -0.017763810232281685, -0.47451189160346985, -1.0977637767791748, 0.2303634136915207, 0.31358808279037476], [-0.915076732635498, -0.5597220659255981, -0.6430894732475281, 0.015026199631392956, -0.1546265035867691, -1.6214741468429565, -0.9330721497535706, -1.3120465278625488, 0.9663652777671814, 0.47680583596229553, -2.2271957397460938, 0.3869367837905884, -0.6859092116355896, -1.150728464126587, 1.4071294069290161, 1.2344030141830444, -1.661048412322998, -0.6117238998413086, -1.7968053817749023, -0.17989881336688995, 0.6020851135253906, -2.597355842590332, 1.191874384880066, -1.2312862873077393, 1.2129911184310913, -0.667436420917511, -0.7550049424171448, 0.0486588217318058, -0.3228950798511505, -0.34897273778915405, -2.3552896976470947, 0.8384711742401123, 1.1271573305130005, 0.004062815103679895, 1.0269122123718262, -0.14745023846626282, 0.014695730060338974, 1.2903257608413696, -0.6767910718917847, -0.6795313954353333, -0.22961129248142242, -2.063067674636841, 2.3257734775543213, -0.6890761852264404, 1.623049259185791, -0.1490621268749237, -0.1572973132133484, -0.4882054328918457, -0.035407695919275284, -1.06667959690094, 0.42257484793663025, -0.272988885641098, 1.21010422706604, 2.4227654933929443, 0.8225252628326416, -0.8164582252502441, 0.1381750851869583, 1.261991262435913, -2.316868782043457, 0.357517272233963, 0.15594883263111115, 0.6894341707229614, -0.20416423678398132, -0.056739695370197296, 1.56195068359375, 1.3672391176223755, -0.6626750230789185, 1.812546968460083, 1.0882961750030518, -0.35440000891685486, -0.2159290909767151, -1.0521007776260376, -0.4494956433773041, -0.11830468475818634, -1.607726812362671, -0.7333256006240845, 0.5048677325248718, -1.2867668867111206, -0.9602788090705872, 1.2013484239578247, 0.047880902886390686, 0.43991589546203613, 0.10612759739160538, -1.2097400426864624, 1.288475751876831, 1.3062008619308472, -0.6936320662498474, -0.10838758945465088, -0.8614267110824585, 1.804146647453308, -0.47472307085990906, 1.5682177543640137, 0.5796849727630615, 0.1358894407749176, -0.5905248522758484, -0.004685540683567524, 0.3332172930240631, 1.166688323020935, 0.8449606895446777, -0.11358921229839325, -1.0456730127334595, -3.0863678455352783, -0.818361759185791, -0.3650306165218353, -1.377659559249878, -1.6402592658996582, 0.15631644427776337, -1.485771894454956, 0.3093545734882355, 1.3320317268371582, -0.12719692289829254, 0.906043291091919, 0.1312849521636963, -0.7230651378631592, -1.3885619640350342, -0.6857767701148987, -0.7384921908378601, -1.4532586336135864, -0.0653667226433754, -0.6147732138633728, -1.6543594598770142, 0.5797297358512878, 2.1032352447509766, 0.9441189169883728, -0.3122517764568329, -1.4489874839782715, -1.2119661569595337, -0.25144150853157043, 0.05027509480714798, -1.4003504514694214, 2.242206335067749, 1.8547561168670654, 0.9416471123695374, 0.24035358428955078, 1.5909292697906494, 0.7837031483650208, 0.946670651435852, 0.0698712021112442, 0.31099262833595276, 0.04782599210739136, -0.3854893445968628, 0.22078874707221985, 0.17801496386528015, -0.3558000326156616, 1.3935446739196777, 0.11915329098701477, 1.1499875783920288, 0.8646020889282227, 0.373135507106781, -0.2295745611190796, 0.09930402040481567, -1.937721848487854, -0.18829724192619324, 1.002646565437317, 0.6339818239212036, -0.2184915691614151, -0.08158658444881439, 0.5643249750137329, -1.0030817985534668, 0.39826634526252747, 0.7404980659484863, -1.3574934005737305, -0.22609150409698486, -0.254303514957428, -0.1939198225736618, 0.29459983110427856, 0.34681424498558044, 0.011684338562190533, -0.7076869606971741, 0.13152800500392914, -0.45227769017219543, 1.4257147312164307, 0.8607916235923767, -1.2500146627426147, -0.5744810700416565, -0.8562390208244324, 0.38563692569732666, 0.5183102488517761, -0.8417292237281799, 1.5962823629379272, 1.9345595836639404, 0.027046315371990204, 0.08726485818624496, -0.6885061860084534, 1.539736270904541, 0.09431816637516022, -1.9502325057983398, 0.9441673159599304, 1.8056986331939697, 0.35656455159187317, -0.8138458728790283, 0.2937743067741394, 0.3634118139743805, -0.8852124810218811, -2.5626516342163086, 0.5539411306381226, 0.11156689375638962, 0.19516922533512115, -0.8648348450660706, 1.6350655555725098, 1.015203833580017, -0.6954085826873779, -0.288469523191452, 0.43644970655441284, 0.3100939989089966, -0.15660327672958374, -0.7416289448738098, 0.27780869603157043, 1.01860773563385, 0.9545527696609497, -0.7874407768249512, 0.29969605803489685, 0.2888343930244446, -1.7922815084457397, 0.5821380019187927, 1.0950145721435547, 0.3791618347167969, -0.32026243209838867, 0.0411326065659523, 0.3733374774456024, 0.13979405164718628, 0.03231385722756386, -0.10031288117170334, -0.051239654421806335, 0.8741449117660522, -0.8074163198471069, 0.685448169708252, -0.26583734154701233, -1.0083022117614746, 0.7754761576652527, -0.1803198754787445, 1.9330099821090698, -1.1312333345413208, -0.17648541927337646, -0.8244084715843201, 0.32818254828453064, 0.5175414085388184, -0.7147222757339478, 0.32385483384132385, 0.22651337087154388, -0.0840446799993515, 0.22413863241672516, 1.7425326108932495, -0.05578817427158356, 0.6832055449485779, -0.7467070817947388, -1.0144575834274292, 0.731918215751648, 0.014599551446735859, 0.4343554675579071, 1.4434757232666016, -0.7343962788581848, 1.086325764656067, -0.5353235602378845, -0.9923094511032104, -0.9057771563529968, -0.9206599593162537, 0.1509552150964737, 0.6969893574714661, -0.8839057683944702, 0.12128514051437378, -0.2942940890789032, -0.8777285814285278, -0.13124896585941315, -0.1747562736272812, -0.8404853343963623, 1.0090079307556152, -0.19870464503765106, -0.26217415928840637, 0.9299361705780029, -0.10270571708679199, -0.18785667419433594, -0.012584959156811237, 1.9631973505020142, 0.7941285967826843, -1.31343412399292, -0.15884236991405487, 0.8722560405731201, 0.6838516592979431, 0.48044103384017944, 1.3344149589538574, -0.014046718366444111, -0.10631456971168518, 0.17683838307857513, -1.2697750329971313, 1.9104723930358887, 0.763711154460907, -0.9168373942375183, -0.49177709221839905, 0.36557820439338684, -1.155151605606079, 0.3695729672908783, 0.06389427185058594, -0.2748515009880066, 1.2707992792129517, 0.6440555453300476, -0.7687314748764038, 1.2692056894302368, -1.438469648361206, 0.6861123442649841, 0.5093475580215454, -0.44304272532463074, 0.06920173019170761, 0.024882396683096886, -1.441705346107483, 1.989365577697754, 0.03799372911453247, 1.713914155960083, -0.7852057218551636, 0.6608887910842896, 1.2094632387161255, 0.9021581411361694, -1.317055583000183, -0.8658222556114197, -1.1573543548583984, 1.1168264150619507, 1.2548322677612305, -0.5020446181297302, 0.07288795709609985, 0.511454164981842, -0.016278833150863647, 0.8194971084594727, -0.25995904207229614, -1.8776401281356812, 0.1935461312532425, 0.10635485500097275, 1.0412824153900146, 0.19377972185611725, -3.1845853328704834, 0.7845697999000549, 1.617530107498169, -0.7419585585594177, -1.5534930229187012, -0.7782823443412781, -0.6634864807128906, -0.11283847689628601, 1.4524507522583008, -0.12097112834453583, 0.1993456780910492, 0.1970435529947281, -0.6264770030975342, -1.471515417098999, 1.4651447534561157, 0.2076393961906433, -0.8733586072921753, -1.5639667510986328, 1.503729224205017, 0.10116482526063919, -1.1225658655166626, -2.1731648445129395, -1.1359162330627441, 0.9585199356079102, 0.8421459794044495, 0.7532171607017517, 0.9870724081993103, 0.3113372325897217, -1.177608609199524, 0.4303778111934662, -0.049103930592536926, 0.7807055115699768, -0.2746473550796509, -2.872168779373169, 1.35102117061615, 1.6490309238433838, -0.19816598296165466, -0.9483858942985535, -0.7367715835571289, -0.1477251648902893, -0.3417874574661255, 0.11295617371797562, 1.6838573217391968, 0.5931604504585266, -0.34016674757003784, 0.9433820247650146, 0.863426148891449, 0.2852694094181061, -0.8642891049385071, 1.681764841079712, -0.5075371265411377, -0.8508465886116028, 0.49634239077568054, -0.09575454145669937, -0.9945198893547058, 1.0662440061569214, 0.6841433048248291, 1.2018061876296997, -0.7399686574935913, 1.249698519706726, -0.12056364864110947, -0.7605339884757996, 0.13451507687568665, -0.6968652606010437, -0.18840159475803375, -1.1305583715438843, -0.04481108859181404, -2.0262768268585205, -0.24657444655895233, 0.15926499664783478, 0.7423191070556641, 1.5788394212722778, 1.963166356086731, 0.5813664793968201, 0.4114742577075958, 0.017056657001376152, -0.5783838033676147, -1.2703791856765747, -0.8293792009353638, 0.32593613862991333, -0.33300259709358215, 0.7864328622817993, 1.0769271850585938, 0.01658381149172783, -0.22846923768520355, -1.0075676441192627, 1.514157772064209, -0.24947771430015564, 0.033735036849975586, -1.1594178676605225, 0.2574152648448944, 1.1851694583892822, -0.8270524144172668, 0.17889203131198883, 0.6320099234580994, 0.2593904137611389, -0.028895802795886993, 1.0877270698547363, -1.8117274045944214, 1.912249207496643, 0.2104000747203827, 0.6495233178138733, 0.10716969519853592, -0.40463700890541077, -0.5224253535270691, -2.3573100566864014, -0.8079400062561035, -0.5434188842773438, 1.7956582307815552, -0.9237574934959412, -1.0586906671524048, -0.9666123390197754, 0.3855273723602295, 2.3283536434173584, -0.0555838942527771, 1.3442072868347168, 1.1692475080490112, 0.7898983955383301, -0.7816587686538696, -0.40296459197998047, 0.17907454073429108, 0.17623385787010193, 0.5678557753562927, -0.0792953222990036, -0.7320300936698914, -0.4114442467689514, -0.2679446041584015, -0.9501922726631165, -0.8082555532455444, 1.2864069938659668, 0.5032668113708496, 0.6365634202957153, -0.03388695791363716, 0.5901361703872681, 1.6986795663833618, -1.7367479801177979, -2.697082281112671, 1.3711745738983154, -0.6674895882606506, 0.267300009727478, -0.22455450892448425, 1.2396107912063599, 1.627536654472351, -1.0987247228622437, -0.3713962435722351, 1.0852961540222168, -0.25913554430007935, 0.24690233170986176, -0.3876059055328369, -0.4916384518146515, 0.4494386613368988, -0.3779340982437134, 0.2359597235918045, 0.3678063452243805, -0.010712618008255959, 0.5113383531570435, 0.07856860011816025, 1.2750396728515625, -1.6587170362472534, 0.13699476420879364, 1.1801183223724365, -0.5186872482299805, 1.4743797779083252, 1.5878221988677979, 0.44578370451927185, -0.9826025366783142, 0.5204168558120728, -0.472383052110672, 0.9561655521392822, -0.5042622685432434, 1.8137496709823608, 0.45461469888687134, -1.166491985321045, -0.02821534313261509, -0.40734243392944336, 0.974500834941864, 0.9697182178497314, 0.9708179235458374, -2.11087965965271, -1.1291990280151367, -0.3819429278373718, -1.237229824066162, -2.4389467239379883, -0.6115975975990295, 1.8892651796340942, 0.4009113013744354, -1.001142144203186, 0.5882787108421326, 0.7084145545959473, -0.831346869468689, 0.2910080850124359, -0.14604653418064117, 0.3181242346763611, 0.8416063785552979, -0.38213756680488586, 1.7333139181137085, -0.2508575916290283, 2.1952178478240967, 1.1127727031707764, -1.6589398384094238, 1.4210717678070068, -1.1832298040390015, 0.17652615904808044, -2.1914565563201904, -1.563364863395691, 0.16518515348434448, -0.3879622220993042, -0.26980704069137573, 0.4129582345485687, -0.8036496639251709, 0.6867280602455139, -1.0886107683181763, -0.0808301791548729, 1.1852048635482788, -0.8936136960983276, 0.17290456593036652, 0.725189745426178, 0.3011879324913025, -0.7272750735282898, -0.5363094806671143, 0.1836094856262207, 0.16433706879615784, 0.5383930206298828, 0.7378522157669067, 0.6114308834075928, 0.9431673884391785, -0.3969053328037262, 1.6372939348220825, 1.0727347135543823, -0.458960622549057, -0.03409935161471367, -0.4124338626861572, 1.7958651781082153, 0.635786235332489, -0.1007935181260109, 0.7315881252288818, -0.4566713571548462, 0.21304365992546082, -0.5121774673461914, -0.9471896290779114, -0.2526167035102844, -0.2136264443397522, -0.00834172498434782, 1.6556085348129272, 0.9934465289115906, -1.5479283332824707, -0.45211261510849, 1.254024863243103, 1.0784515142440796, -0.2998586595058441, -0.031203333288431168, -1.7391210794448853, -0.739823579788208, -0.6105028986930847, 0.23162420094013214, -0.333282470703125, -0.007657689042389393, 0.5907286405563354, 0.7023192644119263, 0.9555293321609497, -0.24954475462436676, 0.20995338261127472, 1.2082165479660034, 0.14682641625404358, -1.7416183948516846, -0.5414804816246033, -0.1998804360628128, -2.441082239151001, -1.383790373802185, -1.8364976644515991, -1.2107783555984497, -0.9405462741851807, -0.3916604518890381, -0.1352197676897049, -1.9595870971679688, 1.0343799591064453, -0.981724739074707, -0.7627508640289307, -0.8644846677780151, -0.8427611589431763, 0.0006944286287762225, 0.7130168676376343, 1.539716124534607, 0.48927903175354004, -1.3120253086090088, 1.1909674406051636, -0.5081404447555542, 1.1066207885742188, 0.38769978284835815, -2.18802809715271, 0.030985884368419647, -2.166106700897217, 0.4802630543708801, -0.19888722896575928, 0.6879028677940369, -0.8016234636306763, 2.0011348724365234, 2.2291412353515625, 1.0141816139221191, -0.47399187088012695, -1.0697449445724487, -0.22756966948509216, 0.5559349060058594, 0.32212504744529724, -0.24975942075252533, 1.2005683183670044, 0.49029281735420227, 1.2778984308242798, -1.0646711587905884, -2.050196647644043, 0.9598294496536255, -1.3788419961929321, -1.3198390007019043, -0.6702733039855957, -1.7087863683700562, -0.8218634128570557, 2.695970296859741, 2.0772879123687744, -1.0431379079818726, -1.8278999328613281, -0.26629364490509033, 0.7650663256645203, -0.1546826809644699, 1.2421355247497559, 1.8211052417755127, -0.9017835855484009, 0.1743561327457428, -0.8029699325561523, 0.5404341220855713, -0.22000908851623535, 0.4149584472179413, 0.33423709869384766, 0.279928058385849, -0.23305155336856842, -1.0534812211990356, -0.5440219044685364, 1.1296403408050537, -0.8028702735900879, 2.0477359294891357, -0.7080572247505188, -0.33913272619247437, -0.04522653669118881, -0.23023921251296997, -0.5794200301170349, -0.49798962473869324, -1.4157215356826782, 0.9672218561172485, -1.3104076385498047, -0.2752574384212494, 0.8150212168693542, 0.868256688117981, -0.46849772334098816, 0.8061813712120056, 1.6992595195770264, -0.4553370773792267, 0.7199527621269226, 0.6695186495780945, 0.7091712951660156, 0.30591338872909546, -1.8671894073486328, 0.7760392427444458, -0.7969664335250854, 0.2768648564815521, 0.12007579207420349, -0.48559415340423584, 1.0834426879882812, 1.082449197769165, 0.20308586955070496, 0.634244978427887, -1.7608810663223267, -1.5512574911117554, -0.3897656798362732, -0.23171529173851013, 1.826641321182251, 0.06387808173894882, 0.7711967825889587, -0.6647471189498901, -0.8279929161071777, -0.7250083684921265, -1.114369511604309, -0.3385736644268036, 0.22817911207675934, 0.07520826905965805, 0.4101701080799103, 0.3534504771232605, -1.777168869972229, 0.6293666362762451, -0.7914003133773804, -0.24969398975372314, -0.3432234823703766, -0.7139045596122742, 0.09349454939365387, -1.0979827642440796, -0.12565135955810547, 1.4965004920959473, -0.7608160376548767, -1.5702909231185913, 2.094921827316284, 1.1958507299423218, 0.7073633670806885, 0.1209583654999733, -0.6400755643844604, -1.4225050210952759, -0.08243928849697113, -0.8590713143348694, -0.4076879918575287, -0.37137290835380554, 1.673374891281128, 0.5854319930076599, 0.031420689076185226, -0.418713241815567, -1.4299397468566895, -0.16133208572864532, -0.19399189949035645, -0.31017088890075684, 1.4039413928985596, 1.7595720291137695, 0.8433540463447571, -0.3966479003429413, -1.0946849584579468, 1.7324589490890503, -0.9920700788497925, 0.16975124180316925, 0.7592687010765076, 2.057215452194214, 0.15194950997829437, -0.9049509167671204, 1.4361692667007446, -0.21311227977275848, -2.2771146297454834, 0.2582540810108185, -0.7815166711807251, 0.3413855731487274, -0.5058383941650391, -1.3229255676269531, 0.9182963371276855, -0.56052565574646, -1.9343761205673218, -1.5123076438903809, 1.8819421529769897, -0.14851891994476318, -0.3882656395435333, -1.1709892749786377, 0.13137049973011017, -0.8189877271652222, 1.969512701034546, -0.07178031653165817, -1.0027899742126465, -0.00818087812513113, 1.3901278972625732, -0.6458209753036499, 0.5209817886352539, 0.09929413348436356, -2.603586196899414, 1.350901484489441, -0.6012922525405884, 0.3048189878463745, -0.011206047609448433, 0.9072805643081665, -0.44377321004867554, -0.4600464403629303, -0.7340533137321472, 0.35544708371162415, -0.33300694823265076, 0.3401331603527069, 1.0723294019699097, -2.416548728942871, -0.843436062335968, 0.6172513961791992, 2.4125657081604004, 2.1190361976623535, 0.1700512319803238, 0.058933325111866, -0.7320746183395386, -0.1817806512117386, -0.7406770586967468, -0.18365676701068878, -0.02005840465426445, 0.40562400221824646, 1.6205775737762451, 0.9379750490188599, -0.44407662749290466, 1.1090223789215088, -0.964612603187561, 0.36342912912368774, -1.101781964302063, 0.8163579106330872, 1.126716136932373, -0.987714409828186, -0.02455543540418148, 0.8325477242469788, 0.36257225275039673, -0.42211127281188965, -0.32672712206840515, 1.012341856956482, 1.2193517684936523, -0.11904416978359222, 1.4263063669204712, 0.39340391755104065, -0.5833315253257751, -0.4573451578617096, -1.829433560371399, -0.36774975061416626, -0.2737342417240143, -0.6291090846061707, -0.8022682666778564, -0.4660385251045227, 0.11473545432090759, -0.226104736328125, -0.2211509793996811, -0.6024413704872131, 0.8052822351455688, 0.02013498730957508, -0.04993939399719238, 0.024692025035619736, 1.5351495742797852, -1.2016841173171997, 0.42121899127960205, -1.2240666151046753, -0.4455227851867676, -0.9923480153083801, -0.1578725278377533, -0.13569383323192596, 0.5196218490600586, -1.7537412643432617, 0.01544156577438116, 1.5134810209274292, -0.12793956696987152, 1.031651258468628, -1.6149611473083496, -1.674614429473877, 1.1090747117996216, 1.2517445087432861, 0.26442280411720276, -0.07324999570846558, 1.5142086744308472, 0.9160057902336121, 1.1977219581604004, -0.6558322310447693, 1.0495240688323975, 0.2714517414569855, 0.7084704637527466, -1.3537092208862305, -0.635036289691925, -0.9058519005775452, 0.35906901955604553, -0.06300146132707596, 0.36313068866729736, 1.3772451877593994, 0.08717856556177139, -0.7497934699058533, 0.35599496960639954, 1.548675298690796, 0.41079434752464294, 0.9345191121101379, -0.5369415879249573, 0.028001222759485245, 0.7140737175941467, -0.24531356990337372, -1.0907009840011597, 1.8781203031539917, -0.8372622132301331, -1.0721757411956787, -0.2259577214717865, 0.07092839479446411, -0.7036659717559814, -0.44520658254623413, -0.13048219680786133, 0.6681877374649048, -0.2907973825931549, 0.5458084940910339, 1.4329380989074707, -0.9030022025108337, -0.6154786348342896, -0.0939224511384964, -0.02715408429503441, 1.8553425073623657, -0.20422199368476868, 0.12880095839500427, -2.2730824947357178, 0.4077357351779938, 0.5553871989250183, 1.039607286453247, 1.1345899105072021, 0.8623904585838318, 0.9081692695617676, -0.6382690668106079, -0.41074055433273315, 0.09343812614679337, 0.1871185153722763, 0.2275199145078659, 0.8073815107345581, -1.3628990650177002, 0.04287484288215637, 0.7490113973617554, 0.35396718978881836, 0.1259543001651764, 0.270744264125824, -1.2072522640228271, 0.4315981864929199, -0.31805840134620667, -1.1500697135925293, 0.5256673097610474, -0.2955056428909302, -0.38529008626937866, -1.091849446296692, -0.5733034610748291, 0.06741023808717728, -2.05855655670166, -1.8836207389831543, -0.15040364861488342, -0.7679857015609741, -0.13644972443580627, 0.11879587918519974, 0.6687623858451843, -0.07291263341903687, -0.1057770624756813, 0.12261077761650085, -0.6905468106269836, -0.5885166525840759, -1.04788076877594, -0.39334234595298767, -1.4359573125839233, -0.17969410121440887, -1.9153894186019897, -0.7433686852455139, -0.8199006915092468, -1.4088181257247925, -1.0058730840682983, -0.19067257642745972, 0.799837052822113, -0.3331243395805359, -0.396504282951355, -2.799462080001831, 0.7935240864753723, 0.5113734602928162, -1.0086363554000854, 0.31232935190200806, 0.6700311303138733, -1.3237842321395874, -0.6093085408210754, -1.2919596433639526, 0.8618292212486267, 0.2118729054927826, 0.5354883670806885, 0.36014750599861145, 0.06841235607862473, 1.2745676040649414, 1.7389564514160156, -1.003436803817749, 0.9256957769393921, 1.2237728834152222, -0.6808846592903137, 0.30019375681877136, 0.03152604028582573, 0.8310850858688354, -1.022027611732483, 0.013327265158295631, 0.9673019647598267, 0.7539220452308655, 0.0259445421397686, 2.1066157817840576, -0.3891249895095825, -0.4104873239994049, -0.1581714153289795, -0.4556807279586792, 1.2791130542755127, 1.283566951751709, 2.6516687870025635, 0.5052916407585144, -0.02256622165441513, -0.192075714468956, -0.09696134179830551, -0.22658061981201172, -2.1931087970733643, -1.2961108684539795, 0.07176633179187775, 0.3266753852367401, -0.4684089720249176, -0.3040687143802643, 0.9848403334617615, -0.47203195095062256, 0.08211544901132584, -2.1146559715270996, -0.9025580286979675, 2.0370032787323, 1.9380499124526978, 0.6257476210594177, 0.09192124754190445, 0.8264309167861938, 1.4272350072860718, -2.6917002201080322, -0.7682150602340698, -0.022083809599280357, -0.0017735488945618272, -0.6993141174316406], [1.4297324419021606, -1.3241075277328491, -0.07668457925319672, -1.2125155925750732, -0.04201271012425423, 0.20948074758052826, -1.207955002784729, -0.4448588788509369, 0.7993491291999817, -1.1900200843811035, -0.3954549729824066, 0.1577833741903305, -0.23234593868255615, -0.5028644800186157, 0.386042058467865, -0.7532397508621216, 0.4069756269454956, 2.0395896434783936, -0.13459837436676025, 0.1735224425792694, -0.2679425776004791, -1.2723826169967651, -0.10205297917127609, -0.803256630897522, -0.830663800239563, -0.23158791661262512, 1.284881591796875, -1.5365920066833496, 0.7555050253868103, 1.7711975574493408, -1.8612065315246582, -1.288425326347351, 1.2996141910552979, 0.42156803607940674, -0.10788065940141678, -0.007883850485086441, -0.3846554756164551, 1.558333396911621, 0.8672245144844055, 0.6765967607498169, -1.3735160827636719, 0.2826078534126282, 0.11478142440319061, -2.2640628814697266, -0.07885059714317322, 2.313807487487793, 1.1704293489456177, 0.5120396614074707, -1.2763371467590332, -1.480090856552124, -0.03247182071208954, -0.541775643825531, 0.4739671051502228, 2.2217984199523926, -1.864406704902649, -0.0562242791056633, -1.856381893157959, 0.39362984895706177, 1.275886058807373, 0.5266560912132263, 0.4958621561527252, 0.8236653208732605, 1.9054816961288452, -1.4920097589492798, 2.0425913333892822, -0.9854813814163208, 0.3834933638572693, 1.3126428127288818, 1.036821961402893, 0.9546954035758972, -0.8059934377670288, -0.811041533946991, -0.3887866139411926, -0.4280640184879303, -0.6056685447692871, 0.5630265474319458, 0.22370052337646484, 0.5195661783218384, 1.0119068622589111, -0.841881513595581, -1.4169296026229858, 0.1530480980873108, -1.5444303750991821, 0.5284754633903503, -0.8380882740020752, -0.9803739786148071, -2.253533363342285, -1.3271886110305786, -0.8556482195854187, -1.2136213779449463, 0.051808472722768784, -0.12397072464227676, -1.5363116264343262, -0.9848172664642334, -0.2898942232131958, 1.0539920330047607, 0.6715297102928162, 1.6549192667007446, 0.960660457611084, 0.6485339999198914, -0.34188297390937805, 0.49955639243125916, 1.226718544960022, 0.851929783821106, 0.14194872975349426, 1.6099789142608643, 0.7893785238265991, -0.2810834050178528, -1.0457416772842407, -0.12367343157529831, 0.5500150322914124, -0.45522651076316833, 1.4650338888168335, -0.19063062965869904, -0.344787061214447, -1.7726635932922363, 0.7262531518936157, -0.7436528205871582, 0.9256220459938049, 0.5829169750213623, -0.5371547937393188, 0.35429078340530396, 0.8157098293304443, -0.6365844011306763, 0.7867010831832886, 0.04309936985373497, 0.329364538192749, 0.6865407824516296, 0.8067408800125122, -0.33160942792892456, -0.06826289743185043, -0.02913517691195011, -0.6633340120315552, 0.4848044216632843, 0.6105577945709229, 1.2494276762008667, -0.6820436716079712, -0.9356482625007629, 0.5799576044082642, 0.9884349703788757, 1.1615630388259888, 0.42381981015205383, 0.04334194213151932, 0.21978238224983215, 0.06065811589360237, 0.48118510842323303, -0.9023007154464722, -0.4663310647010803, -2.0533623695373535, -0.4238736927509308, 0.017704788595438004, 0.6784449219703674, -0.5632011890411377, -0.6240519881248474, -2.48681902885437, 0.6965286731719971, -0.9606927633285522, -0.8426653742790222, -2.803893804550171, -0.6704307794570923, -0.026938043534755707, 0.8366798162460327, 0.054374586790800095, 0.22544372081756592, -1.7041019201278687, -2.6830873489379883, -0.47242945432662964, -0.5010898113250732, 0.4288661777973175, -0.4083233177661896, 0.4394937753677368, -0.7098953127861023, 1.0913262367248535, 0.49894627928733826, -1.8668701648712158, -0.5542778968811035, 0.1428656429052353, -0.6935403943061829, -1.5167090892791748, 0.06436212360858917, 0.17133378982543945, -0.565898597240448, -0.4356384575366974, 0.1252293437719345, -1.4938231706619263, -0.2484796792268753, 0.8523882031440735, -0.9764742255210876, 1.2568082809448242, 1.069135308265686, 0.24764637649059296, -1.1888145208358765, 0.7834494113922119, -0.5507398843765259, -0.07366271317005157, -0.8107321858406067, -0.4669496715068817, 0.5226148962974548, -0.019819585606455803, -1.2916837930679321, -0.315204381942749, -1.4954074621200562, 1.26884126663208, 2.0923047065734863, -1.1920114755630493, 1.5984278917312622, 0.23280656337738037, -0.10664564371109009, -0.8878934979438782, 0.048658426851034164, -0.6307453513145447, -0.769760251045227, 0.6020991206169128, -2.2033629417419434, -0.25957104563713074, -1.211293339729309, -0.06863284111022949, -0.033746667206287384, -0.061705440282821655, 0.323787659406662, -0.4021129906177521, -0.05402388051152229, 0.6868842840194702, -1.1948158740997314, 1.6465983390808105, -0.2646397352218628, -1.4101625680923462, -0.14478561282157898, -0.6435309648513794, 1.0191019773483276, -2.247736692428589, 0.1752508580684662, -0.9645116329193115, -0.3748010993003845, -0.1597250998020172, 0.16576214134693146, -1.1163026094436646, -1.1663492918014526, 0.18549081683158875, 1.2173229455947876, 0.7531039714813232, 0.434909850358963, 0.348517507314682, 2.395024538040161, -0.08604050427675247, 1.4203513860702515, -0.08304639160633087, 1.0585922002792358, -0.058909427374601364, 2.4971601963043213, 1.0012757778167725, -0.24094189703464508, -0.21007300913333893, -0.05191667377948761, 1.2559289932250977, 0.48562678694725037, 1.433363437652588, 0.19257669150829315, 0.0073292008601129055, -1.0651932954788208, -0.07516594231128693, 0.5285621285438538, 0.27793049812316895, -0.32173657417297363, 0.9666525721549988, -0.6122317314147949, -1.2582050561904907, -1.1328997611999512, 0.7062615752220154, 0.04692754149436951, 1.5708314180374146, 1.349342703819275, 0.9168153405189514, -0.5688976049423218, 2.54575514793396, 0.6253020763397217, -0.19019803404808044, 0.5969619154930115, -0.9912213087081909, -0.7373095750808716, -0.204323872923851, 0.9701657295227051, -0.19519899785518646, -0.576227605342865, -1.8561724424362183, -0.6377974152565002, -0.8184983134269714, 1.5577630996704102, -0.05849149450659752, 0.5334078669548035, 0.13804078102111816, 2.1712701320648193, -1.4783179759979248, 0.6281946897506714, 1.458373785018921, 1.5848594903945923, 1.19212806224823, -0.07121773064136505, -1.5753368139266968, -0.174152672290802, -0.3186475932598114, 0.06203789636492729, -0.35989806056022644, 0.5432348251342773, 0.6690923571586609, -0.3402479887008667, 0.540377140045166, 1.2702637910842896, -0.9037150144577026, 0.45021864771842957, 1.8803802728652954, -0.5040910840034485, 1.1680121421813965, -0.6193721294403076, -0.040919478982686996, -1.585408091545105, -1.365027666091919, 1.7115200757980347, -1.0943440198898315, 0.45998620986938477, -0.9746530652046204, 0.6435444951057434, -0.7875580787658691, -1.4071013927459717, -0.009267759509384632, -1.2786920070648193, -0.27464914321899414, -0.1768541783094406, 0.7763874530792236, -1.1746793985366821, 0.6942009925842285, -0.02307293564081192, 0.35654377937316895, -1.122166395187378, 0.7376985549926758, -0.6990311145782471, -1.3526479005813599, -0.6240065097808838, 2.2556772232055664, -0.24348123371601105, -0.5177334547042847, -0.5352422595024109, 0.4375988245010376, 0.07610336691141129, -1.4526368379592896, 1.2666351795196533, 1.2583240270614624, 0.12874776124954224, -0.722072422504425, -1.002410888671875, 0.16191871464252472, -0.6507116556167603, -0.15013813972473145, -0.5258117318153381, -0.16536027193069458, -0.44544723629951477, 0.5716354846954346, -0.6199739575386047, 1.183487892150879, 1.7977657318115234, -0.22502359747886658, -1.0457805395126343, -2.1395747661590576, -2.588057279586792, -0.5010112524032593, 1.1819498538970947, -1.4788427352905273, -1.302112102508545, -1.1494158506393433, 0.2250347137451172, 0.6782678365707397, -0.7456677556037903, -1.2249979972839355, -0.9049423933029175, -1.3199405670166016, 0.11505267769098282, 1.8633824586868286, 1.1821318864822388, -0.8834902048110962, 0.6984240412712097, 0.732061505317688, 2.257207155227661, -0.11033977568149567, 0.9221293926239014, 0.08126453310251236, 2.0917015075683594, 0.046435125172138214, -0.6481322050094604, 0.13615132868289948, 0.5956252217292786, -0.5495173931121826, 1.5581004619598389, -2.2595605850219727, 0.021678846329450607, 1.7010301351547241, 0.05567679554224014, -1.2848083972930908, 0.2133350521326065, -0.911227285861969, -0.13027217984199524, 0.8858404159545898, 0.5890743732452393, -0.5895612835884094, -0.4165704846382141, -0.9742661714553833, 2.1776962280273438, 0.6895087361335754, 0.8699843287467957, 1.1749401092529297, -0.8663546442985535, -0.6256072521209717, -1.0365406274795532, -0.3071862459182739, -0.7115647196769714, 1.3821531534194946, -0.5275371074676514, -0.455863893032074, -1.7654932737350464, -0.06492575258016586, 1.8648375272750854, 0.025072578340768814, 1.011959433555603, 1.6290260553359985, 0.9291622638702393, -0.6888018846511841, 1.9328594207763672, -1.248165249824524, -0.09381089359521866, 1.2129842042922974, -0.19676841795444489, 0.27150216698646545, 2.8957653045654297, -1.6853853464126587, 1.058564305305481, 1.526300311088562, -1.4896268844604492, -0.46370431780815125, -2.4990289211273193, -0.6666112542152405, 0.6017078161239624, -0.5058110356330872, 0.9482547640800476, -1.14063560962677, -0.9631343483924866, 0.13244929909706116, 1.2624701261520386, -0.10950362682342529, -0.584121584892273, 0.9269557595252991, -1.2778302431106567, -0.5343451499938965, -1.460936427116394, -0.5508375763893127, -1.2590047121047974, -0.7210781574249268, 0.10558440536260605, 0.7823536992073059, -0.036779288202524185, 0.9283900856971741, -0.5838383436203003, 0.18521128594875336, 0.4466105103492737, -1.331186294555664, -0.7666484713554382, 0.3978917896747589, 0.22449928522109985, 0.3147426247596741, 0.24429826438426971, 0.4952598214149475, -0.6345461010932922, -0.156365767121315, -0.4342019557952881, -0.247588112950325, 0.22238324582576752, -0.3877815008163452, 0.3853824734687805, 0.3680938184261322, 0.2059866487979889, -0.5485642552375793, -0.24369220435619354, 0.6352517604827881, -0.46938735246658325, 0.2526412606239319, -0.6620764136314392, 2.4586503505706787, -0.03625769168138504, -0.05494182929396629, 0.21035219728946686, 0.4762376844882965, -1.0045560598373413, 0.7131170034408569, -0.39137688279151917, -0.15745146572589874, 0.051781971007585526, 1.2003294229507446, -0.3547465205192566, -1.3055073022842407, 0.9494341611862183, -0.3898497521877289, 1.555280089378357, -2.1378133296966553, -1.1571487188339233, -0.7712005972862244, 0.005304522812366486, -0.024330846965312958, -0.39402109384536743, -0.5394222140312195, -0.2244340479373932, 0.4979488253593445, -1.2109750509262085, -0.8080871105194092, -0.5481788516044617, 1.2337894439697266, 0.3536471724510193, -0.3724023401737213, -0.3859328329563141, 0.8799362778663635, 0.6055198907852173, -1.9747836589813232, -1.5085405111312866, 0.9828571081161499, -0.6688535809516907, 0.447776198387146, -0.3322097063064575, -1.1106208562850952, -0.5692721605300903, 0.09704647213220596, -0.026058224961161613, 0.837064802646637, -2.5350043773651123, 1.2284085750579834, 2.55747389793396, 0.3781442642211914, -1.9475953578948975, 1.0160191059112549, 0.9182060360908508, -0.937099277973175, 0.7778486609458923, 0.904262900352478, -0.8371397256851196, 1.237794280052185, 1.085058569908142, -0.055355459451675415, -0.8633268475532532, 1.5890119075775146, 0.26669809222221375, 1.5640771389007568, 0.846328854560852, 0.21779301762580872, -0.8011913299560547, 0.6567705869674683, 0.6382687091827393, -0.7506290078163147, -1.3728121519088745, -0.03601193055510521, 0.24839048087596893, 0.1250387281179428, 0.8799487948417664, -0.4926607012748718, 0.45427462458610535, 0.4796745479106903, 0.487433522939682, -1.4465068578720093, -0.056060317903757095, 0.16234217584133148, 1.246071696281433, -1.3513767719268799, 0.2585495710372925, 0.7828636765480042, -1.1993409395217896, 0.027793651446700096, -0.5261973738670349, -2.115448236465454, -0.41152459383010864, 0.21712253987789154, 1.0886412858963013, -0.9365851283073425, 1.540266990661621, 0.5488992929458618, -0.23521845042705536, 2.33794903755188, -0.4428435266017914, -0.7146472930908203, 1.2640833854675293, -1.242639422416687, -0.004554294515401125, -1.6873236894607544, 0.3711124360561371, 0.615187406539917, 0.3011358678340912, -0.9474546313285828, -0.37364688515663147, -0.09316807240247726, 0.6239596009254456, 0.19025355577468872, 0.3894408941268921, -0.2922065258026123, 0.8649360537528992, -0.40762367844581604, -0.6851710081100464, 0.319881409406662, -0.19541575014591217, -0.3410326838493347, -0.5553975105285645, -1.5787729024887085, -1.8076896667480469, 2.081629991531372, -0.5265865325927734, -0.43470752239227295, -1.5302704572677612, -1.0022258758544922, 0.40226539969444275, 0.6084346771240234, 0.3989395201206207, 0.07172121107578278, 0.2507563531398773, 0.5600499510765076, 0.47092175483703613, 0.5161123275756836, 0.4009934663772583, -0.5825045704841614, -0.4619081914424896, 0.4564664959907532, -1.7215626239776611, -0.8999987840652466, 0.3293812572956085, 0.18920189142227173, 0.23091650009155273, 0.4077446460723877, -0.18065616488456726, -1.0200791358947754, -0.2782245874404907, -0.2337229996919632, 0.3040671944618225, 1.5210323333740234, 0.5067054629325867, -0.0732845887541771, 0.8641055822372437, 2.2686872482299805, -0.8756529092788696, -1.642560362815857, -1.6416746377944946, -0.9464762806892395, -0.8758847713470459, 0.8130239248275757, -0.100556880235672, -0.08441125601530075, -1.8627556562423706, 0.285957008600235, -0.07294898480176926, 0.7237628698348999, 0.9477348923683167, -0.4031516909599304, -0.18462836742401123, 0.39276665449142456, -0.5604264140129089, -0.29936549067497253, 0.6552609801292419, 0.06897490471601486, 0.21800971031188965, -0.6775919795036316, 0.7185323238372803, -1.7803871631622314, -2.1541197299957275, -1.5800106525421143, 0.6870461106300354, -1.8565828800201416, 1.024247646331787, -0.6487817168235779, -2.0391995906829834, 0.8598896265029907, 1.7547773122787476, 1.3522629737854004, -1.8611119985580444, 1.4757977724075317, 1.6593596935272217, -0.5822575092315674, 0.3714728355407715, 1.2073482275009155, -0.8325198888778687, -0.2702784836292267, -0.8620157837867737, -0.6388459801673889, -0.3564940392971039, -0.13048529624938965, 0.09944575279951096, 0.1918986439704895, 0.09633604437112808, 0.9844743013381958, -0.4055679142475128, -0.01298900693655014, 0.32059580087661743, 0.1510617882013321, -0.45939901471138, 0.2815701365470886, 1.471524953842163, 0.33913835883140564, -0.10482186079025269, 0.5610239505767822, 0.4216991066932678, -0.26804935932159424, 1.2243566513061523, 0.2819545567035675, 0.422135591506958, -0.9762007594108582, -0.5426470637321472, -1.735644817352295, 0.025403423234820366, -0.9888406991958618, -0.6553853154182434, 0.45301294326782227, -0.059855420142412186, 0.25838446617126465, -1.882635235786438, 1.0287946462631226, 0.22935502231121063, -0.5379300713539124, -1.4753484725952148, 1.227912425994873, 0.39409762620925903, 2.044982433319092, -0.8573870062828064, -0.9138320088386536, 0.6862231492996216, 0.2183973491191864, 0.6426123380661011, 0.2159002274274826, -0.4966904819011688, 0.3253317177295685, 0.8388901352882385, 0.7893284559249878, -0.7983828186988831, 1.1844086647033691, 1.5804933309555054, 2.2903149127960205, 0.22405701875686646, 0.24202463030815125, 0.07794145494699478, -1.260074257850647, -0.5360208749771118, -1.2910346984863281, 0.34998708963394165, 0.6025220155715942, 1.5086480379104614, 0.6494765877723694, 0.3983544707298279, 0.5394760370254517, 2.008126974105835, -0.6290247440338135, -1.1547691822052002, -1.6881059408187866, 1.5746421813964844, -1.8033785820007324, 0.13699595630168915, 0.41300711035728455, 0.0576348714530468, 0.8305146098136902, -0.16058064997196198, -0.6725742220878601, -0.7814570069313049, 1.8535425662994385, -1.879724144935608, 0.0905693769454956, -0.6548701524734497, 1.1402314901351929, 1.1427397727966309, -0.08851633965969086, 0.10645954310894012, -1.1483769416809082, -0.45211857557296753, 0.6128876805305481, -0.11483435332775116, -0.7506945729255676, 1.5838395357131958, -0.9467930197715759, -0.6762677431106567, -0.5650845766067505, -0.7024075984954834, -1.2745784521102905, -0.039215270429849625, 0.5959196090698242, -0.06960967183113098, 0.4306432008743286, 0.28667187690734863, 1.812135100364685, -0.6001859307289124, 0.17104776203632355, 1.7706356048583984, 0.5390171408653259, 0.12483611702919006, 0.7851107120513916, -0.33572760224342346, -0.3657418191432953, -0.09862297773361206, 0.4962913393974304, -1.3415755033493042, -0.21095821261405945, -1.346724271774292, -0.39339232444763184, -0.4836008548736572, -0.3373737931251526, 1.0887881517410278, -2.208766460418701, -0.155003622174263, 1.0121625661849976, 0.5748115181922913, -0.19892571866512299, -0.10496041178703308, 0.9027630090713501, 0.6017746329307556, -2.5690646171569824, -0.8764594197273254, -0.7547345757484436, -1.0035860538482666, 0.40264400839805603, 0.08836585283279419, -0.7365068793296814, 0.9959050416946411, 0.5383251905441284, -0.23018810153007507, 0.9955444931983948, -0.6892015337944031, 0.3750484585762024, 1.06954824924469, -0.8349955081939697, -1.0928597450256348, -1.1940690279006958, -1.4400959014892578, -0.24321068823337555, 0.5748169422149658, 0.09644033759832382, 1.1409059762954712, -0.21206817030906677, 0.0496310330927372, 0.0003610814455896616, 1.919620156288147, 0.7108862400054932, 0.293003112077713, 0.42275986075401306, -0.41696450114250183, 0.24147354066371918, -1.4300785064697266, 1.2771720886230469, -1.3607767820358276, -1.356024980545044, 0.8963803648948669, -0.19734618067741394, 0.3036789000034332, 0.27175334095954895, 0.22210577130317688, 1.1793876886367798, -1.4933946132659912, -0.037045612931251526, 0.43708422780036926, 0.4084186553955078, -0.12424970418214798, 0.44571876525878906, -0.3374420702457428, -1.390061378479004, 1.7819819450378418, -1.3602409362792969, 1.3989994525909424, 1.9653016328811646, -0.6867467164993286, 0.7491855025291443, 1.3205400705337524, -1.2145851850509644, -0.9841850399971008, -0.8359529376029968, -0.9743956327438354, -0.24530038237571716, 0.06405492126941681, -0.5825425386428833, 0.8599796295166016, -0.21979428827762604, 0.7091325521469116, 0.2113921195268631, 1.383233666419983, -1.327608585357666, 0.6041988134384155, 0.04834183305501938, -0.23098737001419067, 0.5114684104919434, 0.5270537734031677, 1.7561284303665161, -1.2075798511505127, -0.6831238865852356, -0.7533812522888184, 0.6255379319190979, -0.0620889812707901, 0.26383528113365173, 0.11944007128477097, -0.06457652151584625, -0.6262422204017639, -0.9665428996086121, -1.7767598628997803, -0.6753312945365906, 0.48593413829803467, 0.967232346534729, 0.26477429270744324, -0.07936292886734009, -1.7026705741882324, 0.7081513404846191, -1.149057388305664, -0.41759833693504333, -0.30206093192100525, 0.1290508359670639, -0.5028427839279175, -1.4129269123077393, -0.1814882904291153, -0.7546762228012085, -1.5921915769577026, -0.5195410847663879, 2.094980478286743, 1.0075726509094238, 0.8945100903511047, 1.0011626482009888, -0.09259501844644547, -0.24033145606517792, 0.09875332564115524, 0.07916920632123947, 1.3640525341033936, 1.6827208995819092, -0.03891465440392494, 1.4963970184326172, -0.10158111155033112, 0.30740368366241455, -0.9006921648979187, -0.6388906836509705, -0.7940983176231384, -0.559711217880249, 1.6332534551620483, -0.7527769804000854, -0.5869219899177551, -0.9660394787788391, 0.11222445219755173, -0.08552510291337967, -0.3056727349758148, -1.1945868730545044, 0.6547767519950867, -0.28421396017074585, 1.8689631223678589, -0.890968382358551, -0.48842382431030273, 0.6861470937728882, -0.37918582558631897, -0.9402609467506409, -0.4822377562522888, -1.6806591749191284, 0.400642991065979, -0.5878632664680481, -0.8545415997505188, -0.4171620011329651, -1.5040339231491089, 2.065704822540283, 0.6260668039321899, -0.7229578495025635, -0.09097010642290115, 0.43454697728157043, -1.0841203927993774, 0.6691259145736694, -0.3360377848148346, 0.37243732810020447, -0.39879369735717773, -2.60689640045166, 0.6096872687339783, -0.1663135588169098, -1.1091126203536987, 0.08234041929244995, 1.7650655508041382, 0.3983227014541626, -0.926816999912262, 0.560783863067627, -0.8009023666381836, 1.3035310506820679, -1.286824345588684, -0.4215184450149536, 0.8219947814941406, 0.957634449005127, -0.9201591610908508, 1.295470118522644, -0.4746103286743164, 0.5055797100067139, 0.8388823866844177, 1.491260051727295, 1.1496518850326538, 0.6937046051025391, -1.1532008647918701, -0.5580706596374512, -0.2976771295070648, 0.10484090447425842, 1.109897255897522, 1.6092312335968018, -0.6076752543449402, 2.9783592224121094, -0.15803655982017517, -1.0507559776306152, 0.8955106139183044, -0.5906833410263062, -1.1408507823944092, 0.9111831188201904, -0.7213879823684692, -0.16707615554332733, 0.19785985350608826, -1.0626640319824219, 0.13177406787872314, -0.9191554188728333, 0.18197651207447052, 0.30114978551864624, 0.9626506567001343, 0.3837207555770874, 0.15235398709774017, 0.5157595276832581, -1.906198263168335, -0.35089197754859924, 0.6114618182182312, -0.019785676151514053, -0.15785427391529083, -0.6275281310081482, -0.35423243045806885, -0.28764989972114563, 0.815275251865387, -0.043066274374723434, -1.3831796646118164], [-2.1768815517425537, -0.11115500330924988, 0.5598257780075073, 0.3221850097179413, -0.48442283272743225, 0.8918494582176208, -0.0747627541422844, -0.5467992424964905, -0.17602987587451935, -0.18482132256031036, -0.5722607970237732, -0.056435082107782364, 0.19523195922374725, -2.0111842155456543, 2.050734519958496, 0.5646543502807617, -0.9122419953346252, 0.6520675420761108, -1.064895510673523, 0.5618075132369995, -1.6408861875534058, 0.32877737283706665, 0.43545788526535034, 1.870832920074463, 2.438105344772339, 2.0554094314575195, -0.44771215319633484, -0.16211900115013123, -2.1669366359710693, -0.20059145987033844, 0.5929471254348755, 1.0464099645614624, 1.1616783142089844, -0.4756156802177429, 2.302121639251709, 0.5409749746322632, -0.7651353478431702, 0.5339033007621765, -1.0970975160598755, -0.765552818775177, 0.6540402770042419, 0.9320200681686401, 0.16920509934425354, 1.3373210430145264, 0.05701207369565964, -0.4607570767402649, 0.3154022991657257, 1.9185035228729248, 0.49103835225105286, 0.8898167014122009, 0.1758171170949936, 1.7938140630722046, -0.7164952754974365, -0.16608159244060516, -0.6751443147659302, 1.0742045640945435, 0.6521757245063782, 0.8007673025131226, 0.4256569743156433, 1.1453877687454224, -0.11799640953540802, -0.36279940605163574, 0.8502208590507507, 0.7142568230628967, 0.3876373767852783, 2.1103222370147705, -0.3238237202167511, 0.6728099584579468, -1.457870364189148, 1.2102794647216797, 1.5612475872039795, 0.3205466866493225, 0.35179123282432556, 0.20696260035037994, 0.36061912775039673, 1.0217550992965698, -1.634979248046875, -0.2628801465034485, 0.18856830894947052, 1.178504467010498, 2.1626787185668945, 0.5914912223815918, -1.039209246635437, 0.47639331221580505, -1.307180404663086, -0.2426699846982956, -1.0259162187576294, 1.3349738121032715, -0.38654008507728577, 2.616741180419922, 0.6362025141716003, 1.2709687948226929, -1.2236361503601074, -2.685980796813965, 2.0381572246551514, -1.5273631811141968, -0.14232246577739716, -0.6364738941192627, 1.7190507650375366, 0.7655336856842041, 0.8779363036155701, -0.14733916521072388, 0.5198718905448914, 0.15598249435424805, -0.6022071242332458, 2.202141284942627, -0.9091441631317139, -2.8161816596984863, -0.07703666388988495, 0.8577088117599487, -1.0427027940750122, -1.572435736656189, 1.8020269870758057, -0.6498997211456299, -1.4570543766021729, 0.3894014358520508, -0.25544288754463196, -0.33629652857780457, -0.758484959602356, 2.0697264671325684, -0.3306257128715515, -0.6432045698165894, 0.7669792771339417, 0.2677638828754425, 0.5107524991035461, -0.2764655649662018, -1.139892578125, -0.23806199431419373, -0.22821339964866638, -1.8396022319793701, -0.9206430912017822, -0.18236391246318817, 0.995020866394043, 0.9067640900611877, -0.5196483135223389, -0.3233073651790619, -0.09666537493467331, -0.9391811490058899, 1.1212632656097412, -1.7447947263717651, 2.081758975982666, -0.7913766503334045, 1.1976529359817505, 0.10200414061546326, -1.3407129049301147, -1.9838671684265137, -0.7243750095367432, -0.43683135509490967, -0.3695943355560303, -0.10404215008020401, -0.03734501451253891, -0.2941781282424927, -0.5871824026107788, 1.409303903579712, -0.4307427704334259, 0.2723523676395416, -0.7268190383911133, -0.4275248050689697, 1.0141642093658447, 0.7317609786987305, -1.4713337421417236, 1.8097635507583618, 0.487078994512558, 1.4853050708770752, -1.0697017908096313, -0.36612817645072937, 0.4946053624153137, 0.026825377717614174, 0.2246759533882141, -1.2775492668151855, -0.516032338142395, -0.24720095098018646, -0.633650004863739, 1.224804162979126, -0.26502227783203125, -0.4126310348510742, -0.46826261281967163, -1.9110978841781616, 0.17730601131916046, 1.3714722394943237, 1.9901491403579712, 0.38002899289131165, 0.3635331094264984, 0.24287255108356476, 0.5120249390602112, 0.15028183162212372, -0.7690392136573792, 0.3718089461326599, 1.3843430280685425, 0.49160221219062805, -0.20154349505901337, -0.1613590121269226, 0.13908065855503082, -0.8444744944572449, -0.49737825989723206, 0.17637105286121368, -0.2483261078596115, 0.6627746224403381, -0.677922785282135, -1.3594789505004883, 0.004433312453329563, 0.4866276681423187, 1.987304449081421, -1.4399033784866333, 0.12688513100147247, -0.18022751808166504, -0.4719703793525696, -1.1086792945861816, 0.7167307138442993, 1.535608172416687, -0.6996431946754456, -1.0756176710128784, 0.0304632056504488, 1.2574061155319214, -0.5264885425567627, -0.06429683417081833, -1.1239495277404785, -0.23752640187740326, -0.7762301564216614, -1.6476263999938965, 0.8943497538566589, 0.9251987934112549, -1.722801685333252, -0.25979045033454895, 1.1508779525756836, 0.5464417338371277, -1.5498969554901123, -1.8775776624679565, 1.0332353115081787, -0.4037114381790161, -0.2541387975215912, 1.2322081327438354, -0.609760582447052, 0.67217618227005, 0.06449970602989197, 0.8553690314292908, -0.4588124752044678, 0.2999594509601593, 0.23255655169487, 1.3995915651321411, 0.550880491733551, -0.06039358302950859, -0.1865442544221878, 0.9267376065254211, -0.5512531995773315, -1.1175026893615723, -0.2667448818683624, 0.5986104607582092, -0.39458024501800537, -1.5495517253875732, -0.22655487060546875, 0.072766974568367, 0.4114823341369629, 0.1995706558227539, 0.5676357746124268, 0.7268566489219666, -0.8521323204040527, 1.3115776777267456, -0.05483470484614372, -1.364902377128601, 1.1658128499984741, -0.7540655732154846, 1.3908870220184326, -0.037431005388498306, -0.06204759702086449, -1.5854723453521729, 0.08465367555618286, 1.162667155265808, -1.1250789165496826, 0.19176577031612396, -1.6781622171401978, -0.18746601045131683, 0.7834126353263855, -0.7537508606910706, -0.7366169691085815, 0.6073349118232727, -0.40219026803970337, 1.087988018989563, 0.39813032746315, -0.3929508328437805, -0.1971333771944046, -0.03898748382925987, 0.9300308227539062, -0.16801482439041138, -0.08621940016746521, -0.576208770275116, -0.6112485527992249, -0.025686141103506088, -0.05457897484302521, -1.2139852046966553, -2.3400447368621826, 1.0841350555419922, 0.05764840915799141, 0.6450457572937012, 0.5532613396644592, 0.11342878639698029, -0.3229582607746124, 1.2779812812805176, -0.06036824360489845, -0.7547526955604553, -0.33787551522254944, 1.4127137660980225, -0.5249212980270386, 0.41291218996047974, 0.9791569113731384, -1.3573155403137207, 0.8367486000061035, 0.08351404964923859, 0.021396726369857788, 0.3308890163898468, 0.5616263151168823, -2.578538656234741, 0.004133786540478468, 0.35856133699417114, 0.3791852593421936, 0.028845127671957016, 2.0108189582824707, 0.553916871547699, -0.07859653979539871, -0.3797631561756134, -0.6809875965118408, -1.6280856132507324, -1.1943343877792358, 0.5187228918075562, -0.4693211019039154, 0.39443016052246094, 0.3754734992980957, -0.7142453193664551, -0.7909669280052185, 0.6126985549926758, -1.4697364568710327, -0.41306421160697937, 0.7569557428359985, 0.620977520942688, -1.531751036643982, -0.0567726269364357, 0.7469468712806702, -0.842361330986023, 0.5577667355537415, 0.013176275417208672, 0.4843238592147827, -1.133790373802185, 0.6677957773208618, -1.1396598815917969, 0.43010881543159485, 1.3543168306350708, -0.6585254073143005, 0.8524566888809204, 1.1424036026000977, 0.4059945344924927, -0.5556035041809082, -0.7732678055763245, -0.9041536450386047, 0.704068124294281, -0.3467344641685486, -0.2842766344547272, -0.7105388641357422, -0.5220631957054138, -0.035218700766563416, -1.4044848680496216, 1.19767427444458, 0.35093095898628235, -1.7745730876922607, 1.0922551155090332, -1.4867360591888428, 0.28174301981925964, 1.7317098379135132, 1.5986167192459106, -0.5775305032730103, 1.4864251613616943, 0.39370197057724, 1.4193825721740723, 0.15989094972610474, -0.44118425250053406, 0.3769920766353607, -0.5283346772193909, 0.22933396697044373, 0.6350298523902893, -1.0755722522735596, -0.17651546001434326, 1.2114282846450806, -1.06879723072052, -1.075851559638977, -2.19781231880188, -0.3765403628349304, 0.7841854095458984, -1.1416146755218506, 2.1771039962768555, 0.02395976334810257, 1.0040278434753418, 0.6712915897369385, 0.9359084367752075, 0.012780380435287952, 1.106884241104126, 1.2345036268234253, 0.3007485270500183, 1.022127628326416, 0.735953152179718, 0.13553443551063538, -0.2796296179294586, 0.10558458417654037, -0.10210385918617249, -2.569657325744629, -1.5407840013504028, 0.09512320160865784, -0.08456986397504807, -1.1337575912475586, -0.17941410839557648, -2.380824565887451, -2.807135581970215, -0.6765933632850647, -1.6385293006896973, 1.6165910959243774, -0.48564302921295166, -0.06793217360973358, -0.13710594177246094, -0.813618004322052, 1.4320226907730103, 0.049290768802165985, -0.22782284021377563, -0.14469289779663086, 0.4003790020942688, 0.7719448804855347, 1.4290958642959595, 0.29648929834365845, -0.3156518340110779, -0.31574323773384094, 0.21370212733745575, -0.6717305779457092, -1.0490950345993042, -1.0455021858215332, 0.32295921444892883, -1.1187002658843994, 1.4617550373077393, 1.2446057796478271, -0.08066654950380325, -0.3087685704231262, 0.23321881890296936, 1.4058051109313965, -2.0062177181243896, 0.5306864380836487, 0.12160751968622208, 1.2517231702804565, 1.442126750946045, -1.3354599475860596, -0.40732982754707336, 0.5060594081878662, -0.9216601848602295, -0.352373331785202, 0.3578205704689026, 2.8397269248962402, -0.6262118220329285, 0.3605823814868927, 0.5166448354721069, 2.21690034866333, 0.10980956256389618, -0.2682635188102722, 1.1366833448410034, -1.0553442239761353, 0.49772584438323975, 1.7965008020401, 0.06450492888689041, -1.1537079811096191, -1.1772377490997314, 0.07839055359363556, 0.009589679539203644, 0.9882519841194153, 1.4169548749923706, -1.9679540395736694, 0.4682408273220062, 0.7050676345825195, 0.3932817280292511, 0.8086620569229126, 1.0277317762374878, 0.46398282051086426, -1.2365201711654663, 0.7842810750007629, 0.11265753954648972, 0.5707777142524719, -0.059577737003564835, 0.36400219798088074, -0.345173716545105, -0.3204609155654907, -0.018215686082839966, 0.7265534996986389, -1.1371283531188965, -0.9192370772361755, -1.0838710069656372, 1.1907823085784912, -0.06866875290870667, -1.1732662916183472, 1.277527093887329, -0.48992452025413513, 1.5432593822479248, -0.7471610903739929, -0.37243860960006714, 0.8378671407699585, 0.9673140048980713, 0.9140933156013489, 0.5633322596549988, -1.2960612773895264, -2.0145678520202637, -0.003632803214713931, 0.8247219920158386, 0.9085060358047485, -0.957532525062561, -0.5283300876617432, -1.3134514093399048, -0.730148434638977, 1.5415719747543335, -2.219667673110962, -0.4609786570072174, -1.1265569925308228, -0.7345969676971436, -0.4039718210697174, -1.1978108882904053, 0.6421257257461548, -1.1870845556259155, -0.5754775404930115, -0.2534867525100708, -0.7349068522453308, -0.13409242033958435, 2.178847312927246, 0.1510767787694931, -1.5901209115982056, -1.1391394138336182, -1.3278111219406128, 1.4851953983306885, -0.8196401000022888, -0.17930281162261963, 0.9718806743621826, -2.448594331741333, -0.44420287013053894, 0.6535874605178833, 0.11866115778684616, -0.5719529986381531, -0.4136830270290375, 0.8126564621925354, 1.1613891124725342, -0.7067342400550842, -0.597355842590332, 1.5898438692092896, -0.594438374042511, -0.578718900680542, -0.7623625993728638, -0.009004797786474228, 1.1795817613601685, 0.6729747653007507, 0.42565256357192993, 0.6348022222518921, 1.5491664409637451, -1.2331926822662354, 0.1879511922597885, -0.3915412127971649, 0.2571203112602234, -0.6330260038375854, -0.12514829635620117, 0.2622952163219452, 0.8348091244697571, -0.8808156847953796, -0.8584009408950806, 1.1337705850601196, 0.3942725658416748, -1.460641622543335, 1.9099884033203125, -1.8341565132141113, 0.8786831498146057, 0.8271742463111877, -1.1186894178390503, 1.0894991159439087, -0.9260315299034119, -1.432098150253296, 0.1326308250427246, 0.905423641204834, -1.319785475730896, -1.1245464086532593, -1.1123690605163574, 0.6609976291656494, -1.0025684833526611, -0.22983750700950623, -0.7723481059074402, -0.5442915558815002, -1.055680274963379, -0.8384512066841125, -0.9534690380096436, -0.20621879398822784, -0.08383598923683167, -1.251564860343933, -3.0218334197998047, -0.6046242713928223, -1.6853649616241455, -0.19354134798049927, -0.3575761020183563, -1.010206699371338, -0.12309308350086212, 0.858578622341156, 0.5430202484130859, -1.9505033493041992, 0.9524874091148376, 0.5112136006355286, 0.6779019236564636, 1.6268339157104492, -2.494833469390869, -0.4784950017929077, 1.0058493614196777, -1.1401759386062622, 0.43625953793525696, 1.6483614444732666, -1.5345467329025269, 0.6873436570167542, 0.7740629315376282, -2.175644636154175, -0.1883133202791214, -0.5294457077980042, -0.16443601250648499, 1.6459934711456299, -1.2910336256027222, -0.9480557441711426, -0.8220256567001343, -0.12600766122341156, 0.1056690588593483, 2.0177698135375977, -0.7796466946601868, 0.2738572061061859, 0.6747046709060669, 0.34194281697273254, -0.47015562653541565, -0.8292198181152344, 0.20792239904403687, -0.2718222737312317, 0.542516827583313, -1.0841622352600098, -0.09229796379804611, 1.4965566396713257, 1.5104124546051025, 0.3856850862503052, -0.38083916902542114, 0.4963964521884918, 0.6077818870544434, 1.131761074066162, -0.066636823117733, 1.6996028423309326, 0.5104681253433228, 1.2890290021896362, -0.34962302446365356, 0.5616595149040222, 0.07989692687988281, 0.06817883253097534, -0.1840856820344925, -0.7970296144485474, 1.6541441679000854, 1.6634958982467651, 0.7260835766792297, -0.11602967977523804, -1.6197682619094849, 0.21379992365837097, -0.6959884166717529, 0.470735102891922, 0.42815613746643066, 1.6884573698043823, -0.10292646288871765, 0.0490390919148922, -0.5898112654685974, -0.8943477272987366, 0.7624998688697815, -1.173154592514038, 0.5773739218711853, 0.44476574659347534, -0.48455825448036194, 0.5777308940887451, 0.8588754534721375, -2.600177764892578, 0.6230210065841675, -0.5086202025413513, -2.2509243488311768, -1.4217214584350586, 1.1145789623260498, 0.0333334244787693, -1.5624866485595703, -0.14189960062503815, 0.13395914435386658, 0.9721519351005554, 0.9905354976654053, -0.511380136013031, -0.3826088607311249, -0.44112628698349, -1.2870837450027466, -1.6019936800003052, -0.8069173097610474, 0.7868696451187134, -1.2935140132904053, -0.7109773755073547, 0.20770955085754395, 0.5648391842842102, -0.6880368590354919, -2.0470798015594482, 0.016022859141230583, -0.361013799905777, -0.7504951357841492, -0.7819401025772095, -0.8016846776008606, -0.6283977627754211, -0.2598129212856293, -0.3610369563102722, 0.20322805643081665, -0.17909128963947296, 1.1970254182815552, 1.5485833883285522, -1.7707070112228394, 0.2377784252166748, 1.957892894744873, -1.2612818479537964, -2.5818912982940674, 0.273056298494339, 0.5348877310752869, 0.3645051121711731, 1.0441688299179077, 0.03592423349618912, -0.5697392821311951, 0.14726713299751282, 0.2845115065574646, 0.006441702134907246, 0.29800674319267273, -0.7343156933784485, -2.065944194793701, -0.670759916305542, -0.01662510633468628, -1.3657810688018799, 0.6735572814941406, 0.4942995011806488, -1.012331485748291, 0.07816922664642334, 1.1865226030349731, -1.2083512544631958, -0.6120135188102722, -2.0172348022460938, -1.0247938632965088, -1.4419336318969727, -1.7057907581329346, -0.21735313534736633, -1.1137943267822266, -0.8748973608016968, -0.03231244906783104, -1.6660535335540771, -0.17562133073806763, 1.6015348434448242, 0.4353495240211487, -0.08569833636283875, 1.125754714012146, -0.0813557505607605, -0.01987011358141899, 1.8550070524215698, 0.6788875460624695, -0.0209233146160841, 1.8838417530059814, 0.6090070605278015, -0.9335929155349731, 0.5154531598091125, -2.0289087295532227, 0.03802448511123657, 0.8081944584846497, -0.8216565847396851, 2.188023567199707, -1.0838791131973267, 0.22170577943325043, -0.5803924798965454, 1.1173745393753052, 0.7562341094017029, 1.439452886581421, 0.6634500026702881, 1.352757215499878, -0.637567400932312, 0.07238437235355377, -1.3181904554367065, 0.8161328434944153, 0.9589625597000122, -0.4066500663757324, -0.17231987416744232, -1.3186832666397095, -0.8316344618797302, 0.69439297914505, 2.015289545059204, -0.7980684041976929, 0.3466595709323883, 0.9531506896018982, 1.8168081045150757, -0.584265947341919, -0.17451073229312897, 0.35494545102119446, -0.19370810687541962, -0.6240093111991882, 0.4501613676548004, 0.6279321312904358, -0.9204970002174377, -0.13441379368305206, -0.970365047454834, 1.2454026937484741, -0.061413899064064026, 0.976436972618103, 0.9810411334037781, -1.2148622274398804, 0.4117947220802307, -1.270555853843689, 0.3230926990509033, 1.783893346786499, -2.095047950744629, -1.901675820350647, 0.5555468201637268, 0.19858339428901672, 1.8700227737426758, 1.3600022792816162, 0.9576530456542969, 0.32837948203086853, 1.0910054445266724, 0.6052336692810059, -0.5610098838806152, 1.1890220642089844, -0.322555810213089, -0.06220470368862152, -0.7063301205635071, 2.465399742126465, -0.5188509821891785, -0.5135856866836548, -0.5282620787620544, -0.740784227848053, -1.1328985691070557, 0.627407431602478, -1.3094940185546875, -0.984527587890625, -0.4674971401691437, -0.47829344868659973, -0.5654016733169556, -2.616168737411499, -0.6137803792953491, 0.45481714606285095, -0.11492361128330231, -0.26307982206344604, -0.015954630449414253, -1.7597768306732178, 1.5007603168487549, 0.9689445495605469, -0.2393101453781128, -0.47968900203704834, -2.5854413509368896, 0.4349216818809509, -1.1708636283874512, 1.2079713344573975, -0.375731885433197, -1.074264645576477, -0.025582991540431976, 0.5304131507873535, 0.42935672402381897, -0.2825527489185333, 0.7687323093414307, -0.5837115049362183, -1.0372309684753418, -0.05370441451668739, -0.5065524578094482, 0.29031580686569214, -0.29003384709358215, -0.5127707123756409, -0.6107067465782166, 1.3665753602981567, -1.0399082899093628, 0.6810361742973328, -0.058983027935028076, 0.6863797307014465, 0.9361752271652222, -1.7738162279129028, 0.7015475630760193, 0.4735758602619171, -0.3678513467311859, 0.538266122341156, 0.4755270481109619, 1.0055042505264282, 0.8883789777755737, -0.6004095077514648, -0.6349149346351624, -0.13856232166290283, -0.45992931723594666, -1.1983920335769653, -0.6258108615875244, 0.22231808304786682, 1.0260432958602905, -1.2331762313842773, -0.4659183621406555, 0.09514319896697998, 0.4487302601337433, -1.2280980348587036, -0.011217324063181877, -0.632784366607666, 0.4466411769390106, -0.8369268774986267, -1.1977345943450928, -0.36568668484687805, -1.1629533767700195, 0.29338276386260986, 0.4379611909389496, 0.7675939202308655, 1.1190110445022583, -0.9719105958938599, -0.3315342664718628, 1.7531383037567139, 0.8680649995803833, -0.6365965008735657, -0.29533636569976807, -0.47484609484672546, -1.499369502067566, -0.2181808203458786, 0.1694176197052002, 0.968181848526001, -1.0963056087493896, -0.36787688732147217, -0.511974573135376, 0.10613206773996353, -0.26413893699645996, -0.7107928395271301, -0.15979743003845215, 0.23883181810379028, 0.8000692129135132, 1.917522668838501, 0.12729941308498383, -1.2110601663589478, 1.3477668762207031, 0.008013083599507809, 0.10440706461668015, 1.1884527206420898, 1.3319212198257446, -0.4993162155151367, -0.27923280000686646, 0.9897395968437195, -0.5877543091773987, 0.7163979411125183, 0.8983743786811829, 1.4169660806655884, 0.31661951541900635, 0.05631561949849129, -0.683484673500061, 0.6982885599136353, -2.2817437648773193, 0.32539448142051697, -1.016476035118103, 1.7354764938354492, -0.5240703821182251, 0.0627998560667038, 0.5158698558807373, 0.4453989267349243, -0.7831345796585083, -0.381924033164978, -0.7734057903289795, -0.31684252619743347, -1.5209927558898926, 0.2115311175584793, 0.558053195476532, 1.4032801389694214, 1.4745935201644897, -0.20214959979057312, 0.6048582196235657, 0.24017733335494995, -0.31024181842803955, -0.30754780769348145, 1.3256583213806152, 0.0825527086853981, 1.1788674592971802, -1.4687241315841675, 0.06226425617933273, 2.042468786239624, -0.5484012365341187, -1.5337547063827515, -0.19623726606369019, 0.8623758554458618, -0.3576849699020386, 0.079933300614357, -0.6112033724784851, 0.4629518389701843, -0.006553808227181435, 0.03316933289170265, -0.0887245163321495, -0.5627656579017639, -0.5486281514167786, -0.0669676885008812, -1.2154819965362549, -0.667873740196228, 0.6213051080703735, -0.3452528715133667, 0.18359214067459106, -0.4656602144241333, 1.3737961053848267, 0.2775588929653168, 0.4404890835285187, 2.4771063327789307, -1.56340491771698, 0.6825557947158813, 0.25310540199279785, -1.8717212677001953, 1.2674751281738281, -1.454998254776001, -0.22464519739151, 0.596373975276947, -0.33387231826782227, -0.07947183400392532, 0.3896317780017853, 0.049866270273923874, -0.47965434193611145, -0.092963807284832, 0.10346946865320206, 0.8981230854988098, 1.0565462112426758, 0.058819547295570374, 1.775209665298462, -1.1111034154891968, 1.1445051431655884, -0.4512674808502197, -1.6258502006530762, 0.47058814764022827, 0.5661388039588928, 1.5153766870498657, -2.0786702632904053, -0.7844513654708862, -0.2599930763244629, 1.5469145774841309, -0.6540803909301758, -1.3014614582061768, 1.0641342401504517], [-2.148611068725586, 1.3327113389968872, 0.1754951775074005, -0.4451788365840912, -0.8466287851333618, 1.5762896537780762, 0.20273934304714203, 1.315826654434204, -1.5396740436553955, -0.43160054087638855, 0.40945106744766235, 0.5703983306884766, 0.5053800344467163, -0.03591885417699814, 0.12733890116214752, 0.7146848440170288, 0.17899814248085022, 0.8282764554023743, 0.17008572816848755, -0.765322744846344, 1.3536248207092285, 0.33575958013534546, -0.27119526267051697, -1.2627885341644287, -1.1462606191635132, -1.8233215808868408, 1.079837679862976, 0.5839672088623047, -0.19239096343517303, 0.7883629202842712, 0.6633651256561279, 1.71017324924469, -1.3986092805862427, -1.200842261314392, -0.7297032475471497, -0.38069063425064087, 0.6632237434387207, -0.6492469310760498, -0.8189651966094971, -0.8396915197372437, -0.13807813823223114, -0.28853073716163635, 0.9943791627883911, -0.43915095925331116, -0.4746131896972656, -0.4996303617954254, 1.6773072481155396, 0.965423583984375, 1.2507750988006592, -0.055690180510282516, 0.875832200050354, -0.8877347111701965, -0.8322263360023499, 0.20278756320476532, -1.4041695594787598, -1.2240852117538452, 0.9187638163566589, 0.08506878465414047, -1.2702921628952026, -1.1783170700073242, -0.5101402401924133, 0.025728745386004448, -1.7406744956970215, 0.589719831943512, 0.6981064081192017, 0.06688166409730911, -0.9938049912452698, -1.641912817955017, -0.43333175778388977, 0.6789510846138, -0.1781904250383377, 0.22843679785728455, -0.26401615142822266, -0.7302983403205872, 0.1760087013244629, 2.0424797534942627, 0.6424145102500916, -0.026243751868605614, -1.2188067436218262, -1.076490044593811, -1.2977246046066284, -0.03244689106941223, 0.606931209564209, -1.1017732620239258, -0.3912487328052521, 0.8177990317344666, 0.5323488116264343, 0.25616759061813354, 0.6176593899726868, -0.7747855186462402, -0.3021230399608612, -1.0714893341064453, -1.1238518953323364, 1.219115138053894, 0.7883105874061584, 0.5421642661094666, 0.5598538517951965, 2.58166766166687, -0.939369797706604, -0.3757873773574829, 0.6015354990959167, 1.303251028060913, 0.4807285964488983, 0.2541552484035492, -0.9193425178527832, 0.7590160369873047, -0.9088072776794434, 0.9477217197418213, -0.21999165415763855, -1.3406375646591187, -0.17005155980587006, 0.13219518959522247, 0.22535806894302368, 0.6214696168899536, 1.087848424911499, -0.50800621509552, -1.4470340013504028, 1.262108325958252, 0.327976256608963, 0.26091670989990234, 1.2326924800872803, 0.21347060799598694, -0.7356677055358887, 0.9423048496246338, -0.32389840483665466, -0.3813357651233673, 1.1150546073913574, 2.217013359069824, 0.03549227863550186, -1.153093695640564, 0.6334766745567322, -0.22568026185035706, 1.3692920207977295, 1.4476741552352905, 0.34719914197921753, 0.9925283789634705, 0.1526910662651062, 1.713072657585144, -1.386713981628418, 1.4388325214385986, 0.4874522387981415, 0.9284189343452454, -0.19609026610851288, -0.6114985942840576, 1.070696234703064, -0.054726533591747284, -1.2251372337341309, 1.3744205236434937, -0.03984422609210014, -0.9235711693763733, -0.785851776599884, -1.089956521987915, 0.926628053188324, 1.1014395952224731, -0.6980133056640625, -1.1900159120559692, 0.003160380059853196, -0.9441845417022705, 0.5210844278335571, -0.2707582116127014, 0.5723715424537659, -0.8285903930664062, 0.6227965950965881, 1.9792636632919312, 0.8639911413192749, -0.8663591742515564, -1.3732119798660278, 1.167807936668396, -0.8376038074493408, 0.03133704513311386, -1.0658762454986572, -0.0917382463812828, 1.266745924949646, -0.9257158041000366, 1.2668204307556152, 1.0088433027267456, 0.16543066501617432, 1.7197657823562622, -0.6292212009429932, 1.325390100479126, 0.8287127614021301, 2.0817065238952637, -0.5114166140556335, -1.3345259428024292, -0.753075361251831, -0.2119448184967041, 0.6112660765647888, 0.1304894983768463, -0.6204915046691895, 0.10801961272954941, -0.9291322231292725, 0.9198575019836426, 1.328326940536499, 1.9967381954193115, -0.2132290154695511, 0.720024585723877, 1.087209939956665, 1.405361533164978, 1.7001032829284668, 1.0714677572250366, 0.442213237285614, 2.0519301891326904, 0.4317019283771515, 0.31160101294517517, 0.7628689408302307, -0.8620681166648865, 0.7359535098075867, 1.3991621732711792, 1.2576218843460083, -2.4114067554473877, -0.5932198166847229, 0.9188839793205261, -0.3432205319404602, 0.07110675424337387, -0.741096556186676, 0.6166335940361023, -0.026251250877976418, 1.6343274116516113, -0.5663729310035706, 1.437427043914795, -2.4859917163848877, 0.5171517133712769, 1.0942484140396118, -0.10600153356790543, 0.5812452435493469, 1.7659155130386353, -0.4225964844226837, 2.733595848083496, 0.7190971374511719, -0.40251123905181885, 0.7527868747711182, 0.4425141513347626, -0.4180517792701721, -0.3557334840297699, 2.016373872756958, 0.26970183849334717, -1.2158809900283813, 0.21333611011505127, 0.21848395466804504, -0.03423914685845375, 0.5943688154220581, -0.26096805930137634, -1.826696515083313, 1.440506935119629, -0.6301507949829102, 0.37267839908599854, 0.28310105204582214, 1.6307579278945923, 0.43597254157066345, -0.6270768642425537, -1.2044658660888672, 0.6758977770805359, 0.876283586025238, -0.8354231715202332, 0.2795088589191437, 1.2717406749725342, -0.2555887699127197, -1.1736046075820923, 0.9779729843139648, 1.5078972578048706, 1.9405415058135986, 0.5797625780105591, 0.29582661390304565, -0.6240834593772888, -1.1805551052093506, 1.4223769903182983, -1.4197728633880615, 1.3098992109298706, -1.1074563264846802, 0.09958260506391525, -0.6882609724998474, 1.266144037246704, -0.14499813318252563, -1.7964961528778076, 1.5389297008514404, 0.4248844385147095, -0.5319992899894714, 0.1470216065645218, -0.08975838124752045, 1.9929622411727905, -0.4790009558200836, -1.0071594715118408, 0.4973442256450653, 0.255002498626709, -0.6630736589431763, -0.05231647938489914, 0.9110525846481323, 1.8001995086669922, -0.8716175556182861, -0.8552992939949036, 0.22503554821014404, -0.8425135612487793, -0.7325302958488464, -0.5538139939308167, -1.4307506084442139, -0.8456255793571472, 0.712969958782196, 0.2495172768831253, -0.13740013539791107, -0.18551810085773468, -0.1050679013133049, -0.42977219820022583, -0.5205742716789246, 1.163464903831482, -1.380957007408142, 1.215036153793335, 0.056612249463796616, -0.022582346573472023, -2.3412678241729736, -0.9500426054000854, -0.4296742081642151, 0.035599175840616226, -0.8864672780036926, 0.8651704788208008, -0.430251806974411, -0.20269227027893066, 0.8117016553878784, -0.7087586522102356, 1.131512999534607, -0.6672937870025635, 0.8206337690353394, -0.5665748119354248, -0.5080810785293579, -0.9231795072555542, -1.5754786729812622, -0.8867857456207275, 1.0010079145431519, -0.2422512322664261, 1.4442914724349976, 1.5534441471099854, -0.1333925426006317, -0.6276838779449463, -0.67930006980896, -2.271979570388794, -1.0191123485565186, 1.188213586807251, -1.8957024812698364, 0.10351916402578354, 1.6132279634475708, 0.30614450573921204, -0.377590149641037, -2.3319649696350098, 0.47569161653518677, 1.0901784896850586, -0.5473867058753967, -0.5282389521598816, 0.2103213667869568, -0.33714592456817627, 0.21827365458011627, -0.6504814028739929, 0.18137727677822113, 1.4811702966690063, 0.2703722417354584, 0.8753582835197449, 0.505747377872467, 2.104034185409546, 0.4580641984939575, -0.019918890669941902, -0.9498490691184998, 0.42343634366989136, -0.554999589920044, -0.3353990912437439, 0.14444178342819214, 0.049264054745435715, -0.7369277477264404, -0.878953218460083, -0.9961331486701965, 0.6057623028755188, 0.051847148686647415, 0.06710720807313919, 0.059112001210451126, -1.1465232372283936, -1.0104914903640747, 2.41066837310791, -0.33851420879364014, -0.591856062412262, -1.612902045249939, -1.197823166847229, -1.0540215969085693, 1.0240669250488281, -1.0749421119689941, -0.2682023346424103, 0.9951058626174927, -0.1386691778898239, 1.1528544425964355, -1.7147974967956543, 1.916905164718628, -2.7735047340393066, 0.5186699628829956, -1.2949105501174927, -1.8910009860992432, 1.0589524507522583, -0.4594208002090454, -1.0214823484420776, -0.9916120171546936, -0.4477866590023041, 0.2863674461841583, -0.29600027203559875, 0.24614891409873962, -3.4139208793640137, -0.07801370322704315, -3.094808578491211, -1.0276055335998535, -0.20972685515880585, 0.1930081844329834, 0.0675976574420929, 0.8386920094490051, 0.16028887033462524, -0.5510203838348389, -0.0985535979270935, -0.34898853302001953, -0.39923590421676636, 0.9405021071434021, -0.6967827677726746, -0.4998745620250702, -0.6040009260177612, -1.37300443649292, -0.16093721985816956, 0.14861920475959778, -0.28632253408432007, 0.08656684309244156, -1.947682499885559, -0.5820260047912598, -0.37503260374069214, -0.3240939974784851, 0.571413516998291, -0.6084436774253845, -0.08292325586080551, 1.4069457054138184, -0.5474216341972351, 0.9049139618873596, -1.216510534286499, 1.6063669919967651, 0.5892403721809387, -0.42054787278175354, -0.3653701841831207, -0.12328659743070602, -0.5013654828071594, 0.3546828627586365, -0.0773317888379097, 0.13126292824745178, 1.4323922395706177, 0.8007683753967285, -1.1640294790267944, 1.2793411016464233, -0.011514673009514809, -0.02405679039657116, 0.3719366788864136, 1.8902537822723389, -1.4173640012741089, -0.7695552110671997, -1.58732271194458, 1.2913193702697754, 2.1443495750427246, 1.8307833671569824, 0.25296974182128906, -1.243868350982666, -0.25172489881515503, 1.4347702264785767, -0.8275322318077087, 1.1205579042434692, -0.2209482043981552, 0.11406338959932327, 0.3202569782733917, -2.200457811355591, 0.9663856029510498, 0.997454047203064, -0.9070658683776855, 0.7213631868362427, -0.30283817648887634, -0.2984694838523865, 0.01834101602435112, -0.04090777784585953, 1.1530544757843018, 1.1266683340072632, -0.22517210245132446, 0.26684603095054626, 0.12479286640882492, -1.2866220474243164, -0.49804410338401794, 0.03353266045451164, 0.7455938458442688, -2.114234447479248, 0.14914147555828094, -0.2722466289997101, 0.24601827561855316, 1.7916852235794067, -1.965101957321167, 0.19938643276691437, 0.5961629748344421, -0.9591023921966553, -2.8514556884765625, 0.16812504827976227, -0.3017019033432007, 0.9408544301986694, 0.7555282115936279, -0.052851352840662, -0.06848224252462387, 0.4859890043735504, 0.507537305355072, 0.8209072947502136, 0.8706725239753723, -0.5848840475082397, 0.7078851461410522, -2.5660219192504883, -0.09845507144927979, -0.9525306224822998, 1.8292208909988403, 0.02556402049958706, -0.9725596308708191, 0.025443950667977333, 0.6938366293907166, -0.0692938044667244, 0.7033807039260864, -0.7492740750312805, 1.3894628286361694, 0.5916560888290405, -0.47455155849456787, -0.11866942048072815, -0.5822897553443909, 2.7483787536621094, -0.6679456233978271, -1.203067660331726, -2.0269718170166016, -1.2034932374954224, 0.3747369349002838, 2.2406208515167236, 0.028425104916095734, 1.6939750909805298, 1.7630252838134766, -1.26126229763031, 0.767113447189331, 1.1334054470062256, -0.9340919256210327, -0.6236080527305603, -0.6616790294647217, -0.13453178107738495, 0.41863611340522766, 0.729804515838623, -0.3884325325489044, -2.566230297088623, -1.6145195960998535, -1.0205875635147095, 0.15493664145469666, 1.7669074535369873, 0.3526121973991394, -1.1323996782302856, 1.0789068937301636, 0.18584543466567993, -1.871986746788025, 0.15965011715888977, 0.5305002331733704, 0.7197456359863281, -1.1315865516662598, -0.8804548382759094, -0.083024762570858, -0.2090112864971161, 0.4276449978351593, 1.6595823764801025, 0.7336840629577637, -0.3774500787258148, 0.3061032295227051, 0.3217647671699524, 0.34381911158561707, 0.281318336725235, -0.26840847730636597, -0.4712845981121063, -0.9740161895751953, 0.33324894309043884, 0.08049698919057846, 1.4917631149291992, 0.28434497117996216, -0.05159729719161987, 0.8460150957107544, 0.6179347038269043, -0.7665422558784485, 0.7176200151443481, -0.7412745952606201, 0.2760104835033417, -0.7844908237457275, -0.5942745804786682, -0.1527058184146881, 0.14130210876464844, -0.2884018123149872, -0.15246184170246124, -0.5330566763877869, -0.5512656569480896, -1.428890585899353, -0.919048011302948, -1.0982760190963745, -1.0541952848434448, 0.1353398710489273, -1.2002298831939697, 0.875154435634613, 1.3876115083694458, -1.4641706943511963, -1.9017622470855713, 0.36398646235466003, -0.4543401896953583, -0.31715935468673706, -1.1816651821136475, 0.0738106369972229, 1.1205146312713623, 0.05970722809433937, -1.3645321130752563, 0.7462978363037109, 0.40497541427612305, 1.0932197570800781, 1.4877536296844482, -1.107048749923706, -0.8664394617080688, -0.9562599062919617, 0.0006196851609274745, 0.5850201845169067, -0.2548220455646515, 1.4430665969848633, -0.628092885017395, 0.5262516140937805, 0.34681904315948486, 0.640060305595398, -0.31971627473831177, -0.39987602829933167, 0.6708520650863647, -0.11481086164712906, 1.6412924528121948, -1.991780400276184, 0.07772531360387802, 0.892484188079834, -0.6122968792915344, -0.020014064386487007, 0.7732910513877869, -0.34245169162750244, 1.5450316667556763, 1.6374871730804443, 0.06046073138713837, -0.5052930116653442, -1.5678554773330688, -1.1371208429336548, -0.4890248477458954, 1.2168753147125244, -0.5200774669647217, 0.7085556983947754, 2.3861005306243896, -0.05394928902387619, 0.12110161036252975, -2.2872278690338135, -1.1764297485351562, 0.7482324242591858, 0.23552292585372925, 0.9906474351882935, 0.2667645215988159, 0.6602235436439514, -1.126237392425537, 1.2379876375198364, -1.6740224361419678, -1.7552458047866821, 0.33613377809524536, 0.9901826977729797, 1.0592575073242188, 0.9086754322052002, 0.5860031843185425, 1.1022614240646362, 0.9987006187438965, 1.0965521335601807, -0.5316094160079956, 0.963058590888977, 0.5035248398780823, -0.9881719946861267, -0.31515613198280334, 0.6713964939117432, -0.9484205842018127, -0.7772901654243469, -0.6820530891418457, -0.39502647519111633, -0.26501545310020447, -1.3633145093917847, -0.3727291226387024, -0.7513810992240906, 0.27879488468170166, 0.9400176405906677, 0.4637749195098877, 0.07825411856174469, 0.0051772757433354855, 0.2570396959781647, 1.654738187789917, 0.03831178694963455, 0.33592426776885986, 0.012949059717357159, 2.4855728149414062, -0.31322982907295227, -0.05179524049162865, 0.9527411460876465, -2.7964580059051514, -0.25673604011535645, 0.6238520741462708, -0.7665303945541382, -0.1774105727672577, -2.4024999141693115, -0.0007011427078396082, 0.9263901710510254, 1.6186755895614624, 0.2763582766056061, -0.5088427066802979, 0.6928744912147522, 1.049022912979126, 0.578090488910675, -0.7689297199249268, 0.9944143295288086, 0.5420867204666138, 1.5107463598251343, -0.06734523177146912, 0.16214951872825623, 1.8002328872680664, 0.2774551510810852, -1.0900919437408447, 0.9762961268424988, -0.4037523567676544, -1.0046080350875854, -1.2853193283081055, 0.2643311321735382, 1.1803507804870605, 0.158610999584198, 0.150448277592659, -0.4295445382595062, 0.13300281763076782, 1.169682502746582, 1.2371225357055664, 2.287240743637085, 1.0678722858428955, 0.5822998881340027, -0.8626139760017395, 1.1454371213912964, -1.2933686971664429, -0.2645169496536255, -0.5261865258216858, -1.5805727243423462, 0.6325534582138062, -1.4645582437515259, -1.228877305984497, -2.081895589828491, 0.7862430810928345, 1.2210625410079956, 0.5255444049835205, 0.7373242974281311, 0.32338380813598633, 1.02275812625885, -0.6934415698051453, 0.11957619339227676, 0.9800846576690674, -0.4483949542045593, 0.9231856465339661, 0.3395763039588928, 0.06306114792823792, -0.9280889630317688, -0.6639190912246704, 0.3096432089805603, 0.8550724387168884, 1.2123051881790161, -1.1826549768447876, 1.8088856935501099, 0.39666062593460083, -0.4039180278778076, 0.6333436369895935, -1.5356462001800537, 0.18829144537448883, 0.35624369978904724, 1.2128629684448242, -0.6375923752784729, -0.8531262874603271, -1.3052592277526855, 0.3908970355987549, -0.7570068836212158, -0.8477157950401306, -0.7919113039970398, 0.0073423320427536964, -1.1048741340637207, 0.3428690433502197, -0.2813864052295685, -0.5561910271644592, 1.238254427909851, 1.1021748781204224, -1.0944963693618774, 0.09720084816217422, 0.32185283303260803, 0.18112905323505402, -0.717683732509613, 0.6395412087440491, 0.34017083048820496, -0.5859465003013611, -0.2901126742362976, -0.9139973521232605, 0.4747585654258728, 0.20231899619102478, 0.7095568180084229, -0.49519339203834534, 2.0674684047698975, -1.6454474925994873, 0.4076225757598877, -0.8796838521957397, 0.8343456983566284, -0.6324312686920166, 1.104660987854004, 2.9222521781921387, 0.9493989944458008, -0.8404582738876343, -0.06713443249464035, -0.5179101228713989, -2.4902045726776123, -0.019502578303217888, 0.1682070642709732, 0.3057269752025604, -0.21677586436271667, -0.6510979533195496, 0.8679127097129822, 1.570914387702942, 0.007554404903203249, 0.3851691782474518, -1.1418808698654175, 0.053471002727746964, -0.45537859201431274, 0.6242203712463379, -0.5721836686134338, 0.2799631357192993, -0.5868621468544006, -0.6874592900276184, 0.4024837613105774, -1.1842135190963745, -0.23253484070301056, -0.6491814255714417, -1.5341227054595947, -0.43563175201416016, -1.7252287864685059, 0.011433226056396961, 1.3128443956375122, -0.21375152468681335, 1.6115052700042725, 1.3667120933532715, -0.2644214630126953, -1.432403802871704, 0.13004396855831146, 0.7203378677368164, -0.6545344591140747, -0.6020672917366028, 1.4733470678329468, 3.1503195762634277, 0.2538500428199768, 0.5242684483528137, -0.46690088510513306, 0.01451759785413742, 1.7184141874313354, 1.4539463520050049, -0.4460236728191376, 0.8315234780311584, 0.3005514144897461, 1.7287731170654297, -1.0801944732666016, 1.2023168802261353, -0.21436652541160583, 0.6703001856803894, 1.2970631122589111, -0.10014546662569046, 1.5224875211715698, 0.9900720119476318, -0.6940864324569702, 0.7000784873962402, -0.4960201382637024, -0.5527429580688477, -0.16295015811920166, -0.09689830243587494, -0.33603158593177795, -0.7421689033508301, 1.4019856452941895, 0.3916563391685486, -1.2735742330551147, -0.41959550976753235, -0.3618662655353546, 2.3922641277313232, 1.2648835182189941, 0.5470705628395081, 0.6083725690841675, -0.21522127091884613, 1.7810993194580078, -0.33511853218078613, 0.7759950160980225, -0.4374919831752777, -1.9146723747253418, 0.5564317107200623, -0.7954033017158508, 1.376037359237671, -0.4223089814186096, 1.737949252128601, 0.621592104434967, 0.852354109287262, 0.034225523471832275, -1.2978355884552002, -0.41981005668640137, 1.2146435976028442, 0.1592991054058075, 0.9760140776634216, 0.47538042068481445, 0.211891308426857, -1.144455075263977, 0.755979597568512, 0.11525306850671768, 0.546654224395752, -0.03929656744003296, -0.08398676663637161, 0.018713748082518578, -0.5686647891998291, -1.0546776056289673, -0.017171472311019897, 0.04479987174272537, -0.2528335750102997, 1.10159432888031, 0.46946021914482117, 0.679937481880188, 0.9602978229522705, 0.04938163980841637, -0.5692720413208008, -0.9048463702201843, 0.26510488986968994, -2.1394472122192383, -0.8576802611351013, -0.47047266364097595, 0.032703615725040436, 1.144762396812439, 0.9590942859649658, -0.6685643196105957, -0.6056231260299683, -0.45913612842559814, -0.23543058335781097, -0.8103415966033936, 0.2683272659778595, -0.7104395627975464, 0.1932564377784729, -1.1291897296905518, -1.172721028327942, 1.0832550525665283, -0.4655759930610657, 0.23218196630477905, -0.1944989264011383, 0.3710567355155945, 0.3898833096027374, -0.11159828305244446, -0.37949585914611816, 1.1215696334838867, -0.04386940598487854, -1.203263282775879, -0.08683354407548904, 0.4874216318130493, 1.0898206233978271, -1.0773330926895142, 1.05241060256958, -0.7513054609298706, -0.04041314125061035, -0.8519098162651062, -0.9576504230499268, 0.1689193844795227, 0.3465196192264557, -1.0347026586532593, 0.10826298594474792, 1.3830392360687256, 0.37221741676330566, 1.2748857736587524, 0.5482580661773682, 1.0499906539916992, 0.9309349060058594, -0.4902360141277313, -0.41128551959991455, -0.1956433355808258, -1.3433787822723389, 1.4547364711761475, -0.49363386631011963, -0.3683633804321289, 0.06456213444471359, 0.5395110845565796, 0.7559904456138611, 0.006998087279498577, 0.9232439398765564, 0.4037873446941376, 0.2279168665409088, 1.8679356575012207, -0.2418927699327469, -0.5316529273986816, 1.6054965257644653, 0.9050300121307373, -0.18778449296951294, -0.2834666073322296, -0.06570664048194885, 1.1036416292190552, 1.3195821046829224, -0.5492306351661682, -0.6228428483009338, -0.14902116358280182, 1.2028480768203735, -1.6288931369781494, -1.3547104597091675, 0.1737634390592575, 0.25944995880126953, 1.5023242235183716, -0.04960491880774498, -1.0350139141082764, -0.039824772626161575, -0.5974860191345215, 0.49547338485717773, -2.2214646339416504, -0.08780920505523682, 1.0271852016448975, 0.4391353726387024, -1.493257999420166, 1.055897831916809, 0.13803517818450928, 0.10918577760457993, -0.9569879770278931, -0.9322680830955505, 0.3147267997264862, 0.5691716074943542, 0.7637690305709839, 0.6200950145721436, -0.3411259651184082, 0.03992495685815811, 0.3989597260951996, -0.666521430015564, 1.1135718822479248], [-1.6286264657974243, -0.5821889638900757, 0.5540114641189575, -1.653599500656128, 0.29189765453338623, -0.5367064476013184, 1.4549598693847656, -0.6258208751678467, 1.8642852306365967, -0.06994974613189697, 0.07013606280088425, -1.1336170434951782, -0.9171072840690613, -0.5126338601112366, -0.3224380314350128, -0.5022915601730347, -0.23333434760570526, 0.282087117433548, -0.030424166470766068, -1.0329164266586304, 1.4929291009902954, 0.4845619797706604, 0.8815785050392151, 0.9787589907646179, 1.6344462633132935, 0.009798694401979446, 1.8704699277877808, -0.34855881333351135, 1.4870680570602417, -0.29404789209365845, 0.0382537804543972, 0.02508467249572277, 0.5028715133666992, 1.3604621887207031, -1.1031346321105957, -0.6817498803138733, -1.678062915802002, 1.5795141458511353, -1.1029566526412964, 0.723563015460968, 0.08418623358011246, -0.8188340663909912, -1.3732259273529053, -0.2801688015460968, 0.6478630304336548, 0.32448163628578186, -0.4811047613620758, -0.7573770880699158, -1.1819756031036377, 0.8224461674690247, -0.6636227965354919, 1.2035022974014282, 0.4938819408416748, 0.4235914647579193, -0.3058500289916992, 1.0473289489746094, 1.9395084381103516, -0.12310068309307098, -1.650723934173584, -1.2207154035568237, 0.48289209604263306, -1.0317206382751465, -0.7859141230583191, 0.45598405599594116, 1.5399181842803955, -0.47976383566856384, 0.784694492816925, 0.7449343204498291, 0.6478456854820251, -1.4056721925735474, 2.829676389694214, 0.013624969869852066, -1.5719918012619019, 0.12525025010108948, -0.0007909216801635921, -0.006333260331302881, -0.02318037673830986, 0.503916323184967, 0.39610710740089417, -0.852449893951416, 0.6124538779258728, 0.6111747026443481, -1.2044426202774048, -0.11902176588773727, 0.7343376874923706, 0.07642423361539841, -1.8368799686431885, 1.3559075593948364, -0.77055823802948, -0.5462943315505981, -0.2835317552089691, 0.43410563468933105, 0.0839066132903099, -0.3294001817703247, 1.824639081954956, 1.4559803009033203, -0.6947829127311707, -0.33316630125045776, 0.964911699295044, -2.4942445755004883, -0.2856161594390869, -0.1676589399576187, 1.77534818649292, 0.018986858427524567, 0.802223801612854, 1.001446008682251, 0.12052804976701736, 0.028873847797513008, -0.42730000615119934, 0.2214982658624649, -0.2077195942401886, -0.2405518740415573, 1.4287811517715454, 1.8777029514312744, 1.0143840312957764, -1.4931788444519043, -0.6706615090370178, 0.9184182286262512, 1.9330931901931763, -0.7317605018615723, 0.2744322717189789, 1.7222530841827393, -0.8092485666275024, -0.8903828859329224, -1.7657992839813232, -1.1177875995635986, 1.2732762098312378, -0.4887904226779938, 1.1981011629104614, 0.4027065336704254, 0.9091607928276062, -0.2868068516254425, -1.2076808214187622, -0.055981412529945374, -0.06945588439702988, 1.5709667205810547, -1.699257731437683, 1.0004509687423706, -0.8195929527282715, 0.8586695790290833, -0.9601781964302063, -0.882501482963562, -0.1680469959974289, -0.022478526458144188, -1.1049319505691528, 1.2731276750564575, 0.013205117546021938, -0.6438412070274353, -0.8946648240089417, -0.5393183827400208, 0.592272937297821, -0.19847877323627472, -0.1809825450181961, -0.20198066532611847, -0.23537564277648926, -0.1884320080280304, -1.1777418851852417, 0.70916348695755, 0.09006720781326294, -1.7793028354644775, 0.14117242395877838, -0.38121217489242554, -1.318027138710022, -0.3869393765926361, -1.2944114208221436, -0.5377968549728394, 1.0152630805969238, -0.06670866906642914, -0.0859285220503807, 2.137561321258545, 0.1659516543149948, -0.056922025978565216, 0.23414205014705658, -1.2853984832763672, 0.6589145660400391, 0.4780465364456177, 1.0522843599319458, -0.8672586679458618, -1.058634877204895, 1.0633666515350342, 1.6048897504806519, -0.46089863777160645, 0.9915613532066345, 0.8491352200508118, -0.1352585405111313, -0.8315369486808777, -0.07601065188646317, -0.3438866436481476, -1.1847236156463623, -0.9156185984611511, 0.4856007993221283, -1.0026313066482544, 0.3637418746948242, 1.9137850999832153, -0.2006891667842865, -0.2357562780380249, 0.10435078293085098, -0.6235520839691162, -0.1528940051794052, 0.9247702956199646, 0.21461182832717896, -1.1291208267211914, -0.24497489631175995, 1.1247410774230957, 0.11860041320323944, -1.3942415714263916, -0.15071727335453033, 0.48068010807037354, 0.840641975402832, -0.33246803283691406, -0.0067878118716180325, -0.05425674095749855, -0.26366573572158813, 1.9789000749588013, 0.5596133470535278, -0.6976808309555054, -1.1219369173049927, -0.07952560484409332, -0.26729872822761536, -0.07452204078435898, -0.5046954154968262, 0.25365570187568665, 2.289600133895874, 0.3077735900878906, 0.8422355055809021, -0.4116668701171875, 3.117527961730957, -0.9055893421173096, 0.3352833688259125, -0.1144598200917244, 0.7317450642585754, 0.5200206637382507, 1.0290526151657104, -0.0902838334441185, 0.06962057948112488, 0.7992241382598877, -0.4931618273258209, -0.9507556557655334, -0.2354096621274948, 0.4085034728050232, 0.4325956404209137, -0.7573924660682678, 0.19780245423316956, 0.4548988938331604, -0.3168851137161255, -0.3847115635871887, 0.15449564158916473, 0.2800089716911316, -0.5261243581771851, 0.8165317177772522, -1.6899758577346802, 1.3062355518341064, -0.3204825520515442, -0.4632188081741333, 0.8594188094139099, 0.4165199100971222, -0.8942490220069885, -1.9174120426177979, 0.8435774445533752, -1.1236196756362915, -0.4613374173641205, 0.6101465225219727, 0.2824864387512207, -1.009595513343811, -0.23824498057365417, -0.6648392677307129, -0.2461232990026474, -1.6016536951065063, -1.0564061403274536, -0.21240803599357605, 0.7912583351135254, 1.2114722728729248, 0.07503988593816757, -0.20389334857463837, 0.9378188848495483, -1.2210971117019653, 1.3939493894577026, -1.066336989402771, 0.9680899977684021, 0.9545958042144775, 1.3761677742004395, 1.5540461540222168, -0.5066121816635132, 0.2451668083667755, -1.0932856798171997, -0.07082907855510712, 0.4495418667793274, 0.6062512993812561, 0.12739992141723633, 1.5649775266647339, -0.8029769062995911, 1.2827259302139282, -0.12424001842737198, 0.7835788726806641, 0.34858477115631104, -0.734077513217926, -1.3534469604492188, 0.44262751936912537, -0.004636178258806467, 2.4990170001983643, -0.4711594581604004, 0.5193762183189392, -1.8579813241958618, -0.8319221138954163, -0.4777962863445282, -0.15498200058937073, -3.1253087520599365, -1.1250534057617188, 0.9540051817893982, -0.5380513668060303, 0.28371453285217285, -0.8672130107879639, -1.8819817304611206, -0.30353495478630066, -0.8529210686683655, 1.4070550203323364, -1.3695939779281616, 0.20593447983264923, -0.38525956869125366, 0.22358913719654083, 2.1014511585235596, 1.5762362480163574, 0.028855713084340096, -0.6992312669754028, 0.5685198903083801, 1.3289556503295898, -1.2943432331085205, 0.456777960062027, 1.999677062034607, -0.521857738494873, -1.4567978382110596, 1.2394742965698242, -0.4018794298171997, 1.5905778408050537, -1.0451000928878784, -0.5637983679771423, -0.8620768189430237, -0.44411104917526245, 0.7889010906219482, 1.5750218629837036, 1.4452087879180908, -0.8610003590583801, -2.1292755603790283, 0.13336175680160522, 0.9067286252975464, 0.6487408876419067, 1.3507791757583618, -0.17852289974689484, 0.4049632251262665, 0.8358717560768127, -0.28784888982772827, 2.0051262378692627, -1.3397434949874878, 0.7461490631103516, 0.15174242854118347, 0.4429931044578552, 1.6228381395339966, -1.3543224334716797, 1.0214619636535645, -0.010367395356297493, 0.21994434297084808, 0.2908480763435364, -0.7807090282440186, -0.7460572719573975, -1.8556339740753174, -0.14038221538066864, -0.7393193244934082, 0.7952162027359009, -0.673267126083374, -1.49276864528656, -0.5471340417861938, 0.22915272414684296, 0.6018489003181458, 0.7166544795036316, 0.4135606288909912, -0.41756582260131836, 0.6236169338226318, 1.7509965896606445, -1.8620412349700928, 1.242761492729187, 1.676903247833252, 0.08577641099691391, -0.8186240196228027, 0.5004379153251648, 0.1424206644296646, -0.445943146944046, -0.29349860548973083, 0.7835903763771057, -1.1913505792617798, 0.5451363921165466, -0.6901932954788208, -0.11297637969255447, 0.25948673486709595, -0.2010730355978012, 0.6862074136734009, 0.9945774078369141, 0.35845622420310974, -0.3770296275615692, -0.4257470667362213, 0.1260000765323639, -0.48176074028015137, -0.4598163962364197, -0.27871841192245483, -1.3449326753616333, -0.0792836919426918, -1.4952244758605957, -1.5555278062820435, 0.2539171278476715, 0.5937182307243347, -1.885665774345398, -1.3430014848709106, 1.8687015771865845, 1.038539171218872, -0.901127815246582, 1.196094036102295, -1.5065630674362183, 1.1734720468521118, 1.70974862575531, -0.2087298482656479, -0.2336871176958084, -1.2080233097076416, -0.09686242789030075, -0.4049641788005829, -2.026134729385376, 0.7855483889579773, 1.3450125455856323, -0.7330767512321472, -1.6019364595413208, 0.6340880990028381, 0.267399400472641, 0.028818489983677864, 0.3327726125717163, 0.7386958599090576, -0.8366442322731018, 0.6739485859870911, -0.17757435142993927, -0.9285321831703186, -1.2002084255218506, 1.5493305921554565, 0.8255043029785156, 0.8315755128860474, -1.5282597541809082, -0.3272135555744171, -0.2328459471464157, 0.6833323240280151, -0.09022532403469086, 0.45909687876701355, -0.8050618171691895, 0.009379982948303223, -0.8443772196769714, 0.5818383097648621, 0.26928162574768066, -2.1927895545959473, 0.3166654407978058, -0.15235298871994019, -1.4355390071868896, 0.11945930123329163, -0.6382275819778442, 0.7452765107154846, -1.7750998735427856, -0.8451378345489502, 0.26917314529418945, 0.3045901358127594, -0.15263067185878754, 2.096343755722046, -0.9441066384315491, -0.7407803535461426, -2.157670736312866, 0.8610129356384277, 0.23910635709762573, 0.5112258791923523, 0.41166889667510986, 0.9918645620346069, 1.4507884979248047, -0.7994502782821655, -0.03572094812989235, 1.6009457111358643, 0.8900325894355774, 0.17984183132648468, 0.29246777296066284, 0.12975621223449707, -0.7404512763023376, -0.4481430947780609, -0.44468259811401367, 0.3002510666847229, -0.22852353751659393, 0.7662264108657837, 1.2220768928527832, -0.6095790266990662, -0.332741379737854, -0.6294594407081604, 0.6538853049278259, 0.23792433738708496, -0.19531390070915222, -1.7124840021133423, -1.6638411283493042, -0.1878100037574768, -0.17124059796333313, -1.7989615201950073, -0.4870894253253937, 0.06633847951889038, -0.8166909217834473, -0.4101710915565491, -0.449585497379303, 1.064975380897522, 0.27325576543807983, 1.5368353128433228, -0.5411452651023865, -0.7172821164131165, 0.8566445112228394, 1.0515010356903076, -1.4099699258804321, -0.4112461507320404, -1.0590136051177979, -0.4180256724357605, -0.6002692580223083, 0.4313957095146179, -0.2645817697048187, 0.9068025350570679, -0.2589000165462494, 0.3252372741699219, -0.6558721661567688, -0.18753179907798767, -0.7776129841804504, 0.9127562642097473, 0.9819678068161011, 0.6175085306167603, 0.48532575368881226, -1.9574698209762573, 0.695257842540741, 1.399649977684021, -0.8701270222663879, -1.8521430492401123, -1.2621020078659058, 0.851290225982666, 1.0113919973373413, 0.2128671407699585, -0.635634183883667, -0.3927096426486969, -0.65770423412323, -1.7306121587753296, 1.6754169464111328, -0.5850861668586731, -0.13485462963581085, 0.4792800545692444, 1.7755603790283203, 0.6141386032104492, -0.2545292377471924, 0.9548969864845276, -0.17642828822135925, -0.01844012923538685, -1.8546667098999023, -0.6683240532875061, 0.505607545375824, -0.0008282221388071775, -1.1357805728912354, 1.811936855316162, -0.8956241607666016, -1.5578333139419556, -0.38775068521499634, 0.9700618982315063, 2.057154417037964, -1.270753264427185, 0.2073076218366623, 0.3922446370124817, -0.1411481648683548, -0.7218298316001892, -0.5962240099906921, -0.4196232259273529, 1.8189553022384644, 0.6143030524253845, 0.5302976965904236, -0.003230739152058959, -0.28447726368904114, -0.41477981209754944, -0.038783807307481766, -2.006829261779785, -0.08190096914768219, 0.4530104100704193, -0.44967421889305115, -0.2845333218574524, 0.010417412966489792, 0.6163197755813599, -0.8398968577384949, 1.1143137216567993, 0.6996389031410217, 0.41767027974128723, -0.799906313419342, -0.1437116265296936, 1.6450183391571045, 0.2785734236240387, -0.8126091957092285, -0.8585206866264343, 0.01278416346758604, -1.645016074180603, 0.3145735561847687, 0.14654289186000824, -0.061247870326042175, -0.06197469308972359, 0.9294978976249695, -0.603636622428894, 0.18318767845630646, 1.422324776649475, 0.42209041118621826, 0.061266977339982986, -0.4814174771308899, -2.137179374694824, 0.6135068535804749, 0.5322877764701843, -0.42872509360313416, 0.7227479815483093, 1.4770264625549316, -0.6026259064674377, -1.5181347131729126, 0.6359553933143616, 0.17298413813114166, 0.38345929980278015, -1.077642798423767, 1.6048041582107544, -1.392647385597229, 0.8723201155662537, -0.8643826246261597, 0.17870968580245972, -1.6169301271438599, 0.5080373287200928, -0.1846025586128235, -0.21103069186210632, -0.4411291480064392, -0.28600582480430603, -0.1515001505613327, -0.45126259326934814, 0.6682631373405457, 1.6663861274719238, -2.1170694828033447, -2.072099447250366, -0.29468607902526855, -0.03489962965250015, 0.6880034804344177, 0.6748836040496826, -1.5427526235580444, -0.14439839124679565, -1.172127366065979, -1.2997108697891235, 1.8283500671386719, 0.3886578679084778, 0.28384387493133545, 0.5519679188728333, -0.5785439610481262, 0.7144963145256042, 0.5944188833236694, -0.9295351505279541, -0.27651461958885193, -0.6729949712753296, -0.2955683469772339, -1.4443598985671997, 0.11091991513967514, -1.058042287826538, -1.5037819147109985, 0.10782167315483093, -1.538001298904419, -0.6805552244186401, -1.3508362770080566, -0.36466485261917114, -1.5689448118209839, 0.8346489667892456, 0.6638626456260681, -0.8208027482032776, -0.601404070854187, 0.4714183211326599, -1.7230908870697021, 1.1797378063201904, 0.2862018048763275, -0.8510485887527466, 1.7073187828063965, 0.6430395245552063, -0.4255729913711548, -1.7220715284347534, -1.1003289222717285, -0.9757168889045715, -1.3787922859191895, 1.2278730869293213, -1.610567569732666, 0.5205942392349243, -0.012179656885564327, 0.6264498829841614, 1.1938238143920898, -1.97127103805542, 1.8587474822998047, -2.6861023902893066, 1.1228684186935425, -0.1393955647945404, -1.1514617204666138, -1.8791793584823608, 0.8285349011421204, -0.32973015308380127, -0.47053393721580505, -0.11181513220071793, -1.2927608489990234, -0.9971944093704224, -0.08266695588827133, 1.2778924703598022, -1.1983543634414673, -0.6401166915893555, 0.26089102029800415, 0.5890049338340759, 1.3417764902114868, 1.585180640220642, 0.6867156624794006, 1.1158818006515503, -1.6769206523895264, -0.7942951321601868, -0.9386250376701355, -0.8546738624572754, 1.3999239206314087, 0.7453296184539795, -0.7922360897064209, -0.5730538964271545, -0.24810439348220825, 0.33670228719711304, 0.4892558753490448, 0.13596872985363007, 1.619849443435669, 0.9479161500930786, -0.9099127650260925, -1.152905821800232, 1.0368609428405762, -0.4061630070209503, -1.631554365158081, 0.19953776895999908, -0.3496650755405426, -0.6970744132995605, -2.160064935684204, -2.0201501846313477, -0.20216278731822968, -1.1036432981491089, -0.2946622967720032, 0.41497188806533813, -1.1580111980438232, -0.05009777843952179, 1.5152064561843872, -0.30925366282463074, 1.2285740375518799, 0.11987639963626862, -0.739000678062439, -1.376981258392334, 1.1789368391036987, -1.4684401750564575, 0.6340377330780029, -1.2053807973861694, 0.8307875990867615, 0.185117706656456, -0.19911377131938934, -0.3517288863658905, 0.6466056704521179, -0.2276482880115509, -0.5676776170730591, -0.7060285806655884, 0.20562003552913666, 1.4458599090576172, -1.0579299926757812, 1.2608975172042847, -0.43829500675201416, -1.5551345348358154, 0.1722206324338913, 0.2068338245153427, -0.18774792551994324, 0.17780771851539612, 0.6294522881507874, -0.05391550064086914, 0.7034425735473633, -0.752431333065033, 0.3030949532985687, -0.5046892762184143, -1.7789843082427979, -0.12656071782112122, -0.6981619596481323, 0.5941416621208191, 0.6851630806922913, -0.7640500068664551, -0.20667776465415955, -0.5880621671676636, -0.7402507066726685, 0.012420437298715115, -1.011723279953003, -1.5741125345230103, 1.190547227859497, -2.5315539836883545, 0.1771484911441803, 0.4418807625770569, 0.012949921190738678, -0.7120038270950317, 1.855518102645874, -0.14670291543006897, -1.6003401279449463, 0.6070989370346069, 0.6013780832290649, -1.4926156997680664, -0.9215027689933777, 0.20438769459724426, -0.9145568609237671, 0.5334646701812744, -0.8339703679084778, 0.41693562269210815, 1.0619524717330933, 0.24135959148406982, 0.392787903547287, -0.5267397165298462, 0.42767134308815, 1.635336995124817, 0.7119081020355225, -0.5725705027580261, 0.22573480010032654, 0.030744748190045357, -0.07990614324808121, -0.9103977084159851, 0.40755099058151245, 0.5737037062644958, 0.2742059528827667, 0.505050003528595, 0.8996170163154602, 1.8544204235076904, -0.09065800905227661, -0.04282132908701897, 0.560519278049469, -0.49834007024765015, 0.582717776298523, 0.46953845024108887, 1.9645090103149414, -0.827296257019043, 0.890608549118042, -2.0171613693237305, 0.9081850647926331, -1.552083969116211, 1.2654402256011963, -0.4601488709449768, 0.8361212015151978, 0.8122193813323975, 0.9832255840301514, -0.8343813419342041, -0.28247517347335815, -1.305944561958313, -0.2974172532558441, -0.20709002017974854, -1.1221941709518433, 0.2886645793914795, -1.0921934843063354, 2.4787757396698, 0.8087955117225647, -1.2091559171676636, 0.8526096343994141, -1.073322057723999, -0.366533100605011, 1.5149203538894653, 1.0867127180099487, -0.4032478630542755, -0.56785649061203, -0.47734302282333374, 0.4831865429878235, -0.7641920447349548, -0.5889222025871277, 0.6308820843696594, -1.3744927644729614, -0.8038637042045593, 0.4760979115962982, 0.23654302954673767, -0.6181774735450745, 0.35118913650512695, 0.23662766814231873, -0.9305732250213623, 0.8578150272369385, -1.0017881393432617, 0.17520762979984283, 1.5174015760421753, 0.06177984178066254, -0.4020710289478302, -0.661956250667572, 0.6044568419456482, -0.10100437700748444, -0.4034482538700104, 0.275033563375473, -0.3217223286628723, -1.6126748323440552, 0.04633232206106186, -1.3121674060821533, 0.9027831554412842, -1.0035427808761597, 1.3211338520050049, -1.4109936952590942, 1.2974114418029785, -0.2685003876686096, 1.2896851301193237, 0.6350458860397339, 0.49428829550743103, -0.31906214356422424, -0.036269016563892365, 1.0558806657791138, 0.6158881783485413, 0.08591784536838531, 0.49949824810028076, -0.6437849998474121, 1.782782793045044, -0.17005929350852966, -1.1867072582244873, -0.9199289083480835, -0.007950465194880962, 0.20964962244033813, -0.27484360337257385, 0.7385048866271973, 0.5530252456665039, -0.18907013535499573, 1.4227412939071655, -0.6422350406646729, -0.23785516619682312, 0.4294189214706421, 0.6818658113479614, 0.017896898090839386, -1.1224251985549927, -0.715751588344574, -0.963408350944519, 0.2336077094078064, -0.09263693541288376, -1.3638983964920044, 0.15401554107666016, -2.0098800659179688, -2.0035383701324463, 0.38986220955848694, -0.864280641078949, 0.5355189442634583, -0.21302767097949982, 0.07598161697387695, -0.5106330513954163, -0.0027642028871923685, 0.344055712223053, 0.69856858253479, -1.288233995437622, -1.4024466276168823, 0.22760453820228577, 0.7916737198829651, -1.024332880973816, -1.1132946014404297, -0.08414620161056519, 1.2172034978866577, 1.7858260869979858, 1.2135789394378662, 2.359208822250366, -0.9249544739723206, -0.10120559483766556, -0.7856208682060242, -1.579671025276184, -1.0387669801712036, -1.012040615081787, 0.547929048538208, 0.12102603912353516, 0.28254878520965576, -0.205872043967247, -0.30228593945503235, -0.5289376378059387, 0.6694282293319702, 0.23202170431613922, 0.5014148354530334, -1.2063440084457397, 0.8597420454025269, 1.6045844554901123, 0.23512159287929535, 0.9448977708816528, -0.4980558156967163, 1.264209270477295, 0.3277641236782074, -1.4658488035202026, -1.5260703563690186, 0.16026927530765533, 0.8988829255104065, 1.1555216312408447, 0.6102911829948425, 0.189801886677742, 1.4817003011703491, 1.2829073667526245, -0.6321098208427429, -0.6158586144447327, 1.326326847076416, -1.823867678642273, 0.03230709955096245, -0.14488963782787323, -1.9449533224105835, 0.5596743822097778, -0.36129891872406006, 0.8939656615257263, -1.0777595043182373, -0.27496132254600525, -1.5529052019119263, 0.4840366840362549, -0.113810233771801, -1.570376992225647, 0.03735394403338432, 1.4120994806289673, -0.2505019009113312, -1.534900188446045, 1.0562993288040161, -1.1994965076446533, 0.26413920521736145, -0.9303591251373291, -0.8295536637306213, 0.2890336513519287, -1.117553472518921, -0.43913236260414124, -0.7419326305389404, -0.0535222589969635, 0.06613599509000778, 0.03986339643597603, -0.018770676106214523, -0.302164226770401, -0.5315806269645691, -0.30476057529449463, 0.18773165345191956, 0.1925661563873291, -0.6604176163673401, 0.4692613184452057, -0.6091535687446594, -2.393784999847412, -0.26500335335731506, 0.24305230379104614, -0.4578813314437866, 0.2009996622800827, -0.35208243131637573], [-0.8571373224258423, -2.337311029434204, 0.05861024558544159, 0.6925775408744812, -0.7454522848129272, 0.40586748719215393, -0.3726964294910431, -2.010190725326538, -1.9793034791946411, 1.643271565437317, -0.20443087816238403, 1.8266160488128662, 0.9640505909919739, -1.1089437007904053, 0.9117987751960754, -1.5423210859298706, 0.6129761934280396, 2.1611435413360596, 0.9925311803817749, -0.5598247051239014, -0.3166448175907135, 0.32495254278182983, 1.1431128978729248, -0.12760378420352936, -0.10605870187282562, -0.9707276225090027, 0.2940801978111267, -0.6921171545982361, 0.6631889939308167, -1.3398538827896118, -0.6580562591552734, -0.26174652576446533, 0.2119806706905365, -0.8563381433486938, -1.6083698272705078, -0.07568515837192535, 0.6740474104881287, 0.1062779352068901, 2.2437593936920166, -1.3008352518081665, -0.5588030219078064, -1.0723838806152344, 0.5438764095306396, -2.818648338317871, -0.8452882766723633, -2.0026955604553223, -0.9412930607795715, -0.4537612497806549, 0.8476046323776245, 0.8079110980033875, 0.624656617641449, 0.2671792805194855, -0.6947342753410339, -0.8260975480079651, 0.375875860452652, -0.7619889378547668, -0.7887924909591675, -0.18529176712036133, -1.645689606666565, -1.0507793426513672, -0.4761027991771698, 0.7094623446464539, -2.4662044048309326, -1.0128127336502075, 1.4678776264190674, 1.4489076137542725, 0.1874815821647644, 0.1903606504201889, 0.09457739442586899, 1.4187685251235962, 0.47301775217056274, -0.2180001139640808, 0.07055540382862091, 0.9206130504608154, -0.8979406952857971, 0.4036356210708618, 0.6749367117881775, -0.8800559639930725, -0.981221616268158, -1.519332766532898, -1.2069711685180664, -1.5007504224777222, 0.34584513306617737, 0.6570740342140198, -2.036787748336792, 0.6095724105834961, 0.8420484066009521, -0.5879018902778625, 0.1625872403383255, 0.6966560482978821, -1.0177528858184814, -0.26734888553619385, -0.7333020567893982, -0.6290242075920105, -0.07435980439186096, 0.7295108437538147, 1.8000754117965698, -1.3860576152801514, 3.3801233768463135, 1.580055594444275, 0.9775190353393555, -0.6831820607185364, 0.1340855211019516, 1.185755968093872, 0.7567766904830933, 1.2180171012878418, -0.09398702532052994, -2.483111619949341, 0.8810131549835205, -0.47802430391311646, 0.6109015941619873, 0.4070371985435486, -0.26794853806495667, 0.33741259574890137, -1.057603120803833, 0.5327529907226562, 1.6862491369247437, -1.2815227508544922, -1.9391496181488037, 0.06369055062532425, 0.15808892250061035, -0.41120150685310364, -0.3961387276649475, -0.1930960714817047, -1.180637240409851, 1.4370756149291992, -1.254088282585144, 1.7358921766281128, -0.044184211641550064, 0.43674954771995544, -0.02116687037050724, 0.14184726774692535, -0.9683588147163391, 0.10042382031679153, 0.9654713869094849, 1.4262531995773315, 0.6398513317108154, -0.2019810825586319, -0.029921896755695343, -1.4225817918777466, -1.8000670671463013, -1.8553165197372437, 1.6664026975631714, -0.48690474033355713, 0.3783452808856964, -0.49167805910110474, 0.8829472661018372, 1.7980746030807495, 1.1979155540466309, 0.6779458522796631, 1.281449317932129, -0.4052073657512665, 0.4760281443595886, -0.33712658286094666, 0.7870868444442749, -0.24726532399654388, -0.5437971949577332, -0.5963887572288513, -0.11686056107282639, 0.5257169008255005, 0.2452651858329773, 2.079375743865967, 0.5669291019439697, -0.6861910820007324, 0.9594834446907043, -0.004765034653246403, 1.0963166952133179, -0.5202727913856506, -2.135659694671631, 1.2632232904434204, -0.09983299672603607, -0.6069356203079224, 0.2175517976284027, -1.0427862405776978, 0.3606390953063965, 0.03327558934688568, -0.8241833448410034, -0.44194507598876953, -3.4528090953826904, 0.2403162121772766, -1.547127604484558, -0.13492676615715027, -0.3049095571041107, 0.15051713585853577, -1.305403709411621, -0.20912091434001923, -0.843386709690094, -0.11235002428293228, -0.2815900146961212, -2.6418278217315674, -1.1296809911727905, -0.3724829852581024, 0.6614562273025513, 0.523012101650238, -1.8538436889648438, -0.604264497756958, 0.17509499192237854, -1.6581206321716309, 1.025429129600525, 2.45975661277771, 0.4750070571899414, -1.3838952779769897, 0.06446616351604462, 0.27670779824256897, -0.3056958019733429, -0.612006425857544, 0.775723397731781, 1.811004400253296, -1.4903342723846436, -1.5012028217315674, -1.7448222637176514, 0.61996990442276, -1.4879063367843628, -0.9902952313423157, 3.1350529193878174, -1.0145701169967651, -0.301876962184906, 0.6207141280174255, 1.4553457498550415, -1.4642372131347656, 0.23632031679153442, 0.583195686340332, -0.4447837769985199, 2.094651699066162, -0.18311215937137604, 0.8885068297386169, 0.5988694429397583, -0.42769625782966614, -1.220732569694519, 1.1059582233428955, -0.28715184330940247, -0.36914581060409546, 1.1109631061553955, 1.0640020370483398, 0.15092773735523224, 0.24964237213134766, 0.5112122893333435, -0.31301552057266235, 0.8216502666473389, 1.2919572591781616, -1.242700457572937, -0.8952351212501526, -2.9954936504364014, -1.3366764783859253, -1.8297613859176636, 0.41748443245887756, 1.5051229000091553, -0.3742485046386719, -2.457430601119995, 0.22550898790359497, 0.4988192319869995, -1.0348302125930786, 1.634359359741211, -0.6248764991760254, -0.659305214881897, -0.08136291801929474, 1.0684980154037476, 1.9109196662902832, -0.3656408488750458, 1.5814063549041748, -0.32869115471839905, -0.7277054190635681, 0.3950231671333313, 0.13289563357830048, -0.7993467450141907, 0.40388917922973633, -0.26295435428619385, -0.9248442649841309, -0.6601157188415527, -0.31934216618537903, 1.8557746410369873, -0.6226220726966858, 0.8461376428604126, -1.0931153297424316, -0.9424960613250732, -0.5528945922851562, 1.5664026737213135, -1.9756288528442383, 0.410412460565567, -1.2423393726348877, 0.5260223746299744, 0.9453150033950806, -0.9594954252243042, 1.402248740196228, 1.4519987106323242, 1.2087169885635376, -0.7647137641906738, -0.971634566783905, 1.1447434425354004, -0.9061062932014465, -1.1352518796920776, 0.33642643690109253, -3.213104486465454, -1.2231343984603882, 0.2233935296535492, 0.7025203704833984, 1.0782579183578491, 0.007079280447214842, 1.1860955953598022, -0.17330247163772583, 0.0026258304715156555, 0.7104485034942627, -1.93672513961792, 0.766211211681366, 0.11666782945394516, -0.8682823777198792, 0.8930023908615112, 0.10078506916761398, 0.32967936992645264, 0.6770461201667786, -0.07767053693532944, -0.6087526679039001, -0.9146990180015564, -0.03514227643609047, -0.23950780928134918, -1.5305616855621338, -0.8456493020057678, -0.154473215341568, 0.6093627214431763, -0.9016368985176086, -0.32586514949798584, 0.8324609994888306, -1.0420135259628296, 1.194319248199463, 0.6149462461471558, 0.26932787895202637, 1.7203582525253296, 1.1499171257019043, -0.9116562008857727, 0.2022322565317154, -0.13783356547355652, -2.2929515838623047, 0.8580251932144165, 1.1105728149414062, -0.951960563659668, -0.5295906662940979, -1.262267827987671, -0.6864595413208008, 1.1336442232131958, -1.934786319732666, 0.2236340045928955, -0.1682879626750946, 0.8424981832504272, 0.017995834350585938, 0.3680729269981384, -0.4383774697780609, -0.059473879635334015, 1.8344790935516357, 0.36190274357795715, 0.9331494569778442, -0.27604278922080994, 0.005001193378120661, 0.5764099955558777, 0.5421409606933594, -0.2725756764411926, 0.6774538159370422, 1.1551908254623413, 0.33670371770858765, 0.7967247366905212, -0.2190517783164978, 0.06751417368650436, 0.11952804774045944, 1.5415257215499878, 0.39110293984413147, 0.7129493355751038, 0.13289755582809448, 0.3317376971244812, 0.44053658843040466, 0.44250452518463135, -0.8445534110069275, 0.5612341165542603, -0.7403601408004761, 0.6394651532173157, 1.12913179397583, 0.8369957208633423, -1.103952169418335, -1.4818989038467407, -0.7864237427711487, -0.4927899241447449, 1.178009033203125, 0.7454935312271118, 0.68641197681427, -0.20759151875972748, -0.032594483345746994, -0.9564453959465027, -1.3360592126846313, -0.30357059836387634, -1.4691964387893677, -1.2772166728973389, 1.2545374631881714, -0.6117440462112427, 0.25334784388542175, 0.17211824655532837, 1.2222352027893066, 0.3998243510723114, -1.4248278141021729, -0.5445438623428345, -0.29021650552749634, -0.7734857201576233, 0.17402279376983643, 0.4362260699272156, 0.9862356781959534, -0.7271170020103455, -0.146993026137352, 0.6741191148757935, -1.6286202669143677, 0.5447578430175781, -0.12778398394584656, -0.5037544965744019, -0.21858389675617218, -0.5896521806716919, -0.08820421993732452, 0.48562389612197876, -0.4695402979850769, -0.21594347059726715, -1.78971529006958, -0.23568691313266754, 0.12082231789827347, -0.5332282185554504, 0.814259946346283, -2.008984327316284, 1.28627347946167, -0.03135918453335762, -0.5001969337463379, 0.025230126455426216, -0.03500928357243538, -0.2694709300994873, 0.8095621466636658, 0.458323210477829, 2.1993558406829834, -1.1412067413330078, -0.9436594843864441, 1.1743576526641846, 2.05769681930542, 0.2507377564907074, 1.5667996406555176, 0.2938994765281677, 0.7104279398918152, 1.117160677909851, -1.0947706699371338, -0.7015547156333923, 0.11703132092952728, 0.0470285527408123, -0.27129873633384705, -0.6744114756584167, 0.5547513365745544, 1.547532558441162, -0.760648250579834, 1.2840850353240967, -0.7089939713478088, -1.853204607963562, 0.6508727669715881, 0.5359480977058411, -1.694244384765625, -0.07756789028644562, -3.1376359462738037, 0.380981981754303, -0.35655897855758667, -0.6903157830238342, 1.1599429845809937, -0.3653677999973297, -2.2289657592773438, -0.42086490988731384, 0.21134257316589355, 0.5082968473434448, -0.10071835666894913, -1.8003562688827515, -0.5204467177391052, 0.7501888275146484, 0.15844248235225677, 0.6492934226989746, -1.2536704540252686, -1.135083794593811, 0.09609360247850418, -0.11964909732341766, 0.3193008005619049, 0.10666442662477493, 0.8243808746337891, 0.5805112719535828, -0.37019360065460205, 0.8555215001106262, 0.42499449849128723, 0.06498513370752335, 0.5280240774154663, -0.7998263835906982, -0.6150439977645874, 0.41723284125328064, -0.015559475868940353, -0.09868595004081726, -0.01522265188395977, -1.0614209175109863, -0.6310154795646667, 1.0499584674835205, -1.3960342407226562, -0.8938944339752197, 0.8184022903442383, 0.43272361159324646, 0.45623594522476196, -0.07913730293512344, 0.3663247227668762, 0.6148459911346436, -1.3762565851211548, -1.539110541343689, 1.033488392829895, -0.44847288727760315, -0.47615399956703186, -0.458626925945282, -0.9625288844108582, 1.002968192100525, -0.01773591712117195, 2.566476583480835, 0.8493463397026062, 0.13073068857192993, -0.6135863661766052, 0.6584100723266602, -0.833869993686676, 0.6710007786750793, -0.9953353404998779, 1.0273418426513672, -2.1738393306732178, -0.48888519406318665, 1.6919420957565308, 0.9596415758132935, 0.941258430480957, 0.6773352026939392, 0.043280407786369324, 0.2991836965084076, -0.01667477935552597, 0.5278255939483643, 0.5828760266304016, -0.5484833121299744, 0.720836341381073, -1.550567865371704, 0.5988520979881287, -0.32693496346473694, 1.1492276191711426, 2.015231132507324, -0.42068326473236084, 0.1865873485803604, 0.21926426887512207, -0.31744009256362915, -1.6956465244293213, -0.1651710718870163, -0.47804903984069824, -0.2433946281671524, -0.7831277251243591, 0.6570457220077515, 0.49998992681503296, -0.9426308870315552, 1.099013090133667, 0.24791507422924042, -0.8498169779777527, -0.36709365248680115, 1.4187347888946533, -0.1695881187915802, -1.0128061771392822, -0.8414009809494019, 2.436361789703369, -1.7994768619537354, 0.6677364706993103, 0.2615923285484314, 1.293091893196106, -0.6632677912712097, -0.4600895047187805, 1.5967040061950684, -0.9942047595977783, 1.3203703165054321, 1.278119444847107, 0.4828711152076721, 1.7942955493927002, -0.5955276489257812, 0.37958765029907227, 0.47796717286109924, -1.7676090002059937, 0.25396671891212463, -1.1466305255889893, -0.2350342869758606, -0.9676856994628906, 0.29392004013061523, -0.2110777646303177, -0.32724544405937195, -1.7810195684432983, -1.256695032119751, -1.90358567237854, -1.5907012224197388, 1.263155460357666, -0.5301703810691833, -0.03373076766729355, -0.7842090725898743, 0.7291531562805176, -0.10730613023042679, -0.05667338892817497, -0.2450466901063919, 0.5380452871322632, -1.5172398090362549, 0.8066858053207397, 1.5195579528808594, -1.027785062789917, 0.10339139401912689, 0.7636783719062805, -1.2046102285385132, -0.9948768615722656, 0.6784371137619019, 0.7711060643196106, -0.008151807822287083, 0.8546162843704224, -0.22353574633598328, -0.3582528531551361, -0.935192346572876, -0.6585158705711365, -0.25274690985679626, -0.09429578483104706, -0.41306281089782715, 0.41577261686325073, 0.0010591564932838082, 0.16964077949523926, 0.991199254989624, -1.3069549798965454, 0.33329811692237854, 1.2363899946212769, -0.3863123059272766, 0.34792450070381165, 0.8206148147583008, 0.40408453345298767, 1.4082649946212769, -0.4697791039943695, -0.043051790446043015, -0.9529374241828918, 0.0005310157430358231, -1.6938343048095703, -0.7381782531738281, 0.20742914080619812, -0.6301411390304565, -0.0877019539475441, -0.35525742173194885, -0.7278075218200684, -0.38001760840415955, -0.031585849821567535, -1.717437982559204, 0.6991549730300903, -0.3447542190551758, -2.0832817554473877, 2.228281021118164, -2.18403697013855, 1.2347486019134521, -0.5745201706886292, 0.1769101470708847, 0.7149028182029724, 0.6618163585662842, 1.5124274492263794, 0.22459174692630768, -0.7276409864425659, -2.13991641998291, 0.48593899607658386, 1.236003041267395, 0.8658353090286255, 2.492089033126831, 0.4362452030181885, 0.7547898292541504, 0.041783254593610764, 0.5139838457107544, 0.20173218846321106, -1.1073698997497559, 0.2894924581050873, -0.06423822790384293, -0.6145122051239014, 1.530950665473938, 0.4631074368953705, -1.5740090608596802, 0.6855451464653015, -0.33210697770118713, -1.1642060279846191, -0.6494966149330139, -0.34184595942497253, 1.5973236560821533, 1.168033480644226, -1.1509206295013428, -0.13597674667835236, 0.023799963295459747, 0.4265744090080261, -0.6851964592933655, 0.12106688320636749, -0.4348418712615967, 1.5277330875396729, -0.033744845539331436, -1.0240694284439087, 0.6427531838417053, -0.04420536756515503, 0.14887553453445435, -0.2901327610015869, 0.09887366741895676, 0.5625685453414917, 0.03568728640675545, -0.8093282580375671, 0.14488860964775085, 0.6859078407287598, -0.14814765751361847, 1.4500318765640259, -1.107041835784912, -2.773245334625244, -0.22583451867103577, -1.0932315587997437, 0.9742556810379028, -1.7390626668930054, 0.29815369844436646, -0.5698109269142151, 0.7919701933860779, -1.1792696714401245, -1.5698894262313843, -1.4950119256973267, 0.49806901812553406, -1.467394471168518, 1.050923228263855, -0.5574891567230225, -0.9352611899375916, 0.5768969655036926, 2.1385421752929688, -1.0435208082199097, -0.9741325974464417, -1.6340965032577515, -0.05788858234882355, 0.3219720125198364, 0.1096905991435051, 0.20029610395431519, -0.25151529908180237, -1.3868145942687988, 0.7203980684280396, -0.8620871901512146, 0.6536493897438049, -0.12146974354982376, 1.630180835723877, 0.6396381855010986, -1.2011407613754272, -1.4153293371200562, -1.6098346710205078, 0.32860293984413147, -1.3898911476135254, -1.081121563911438, -1.3827953338623047, 0.03823062777519226, 0.3671695590019226, 0.3919287323951721, -0.5086565017700195, 1.033513069152832, -1.3356176614761353, -0.8381302952766418, 0.7002349495887756, -0.9737635850906372, -0.42359092831611633, 0.903416097164154, 0.12676222622394562, 1.5615476369857788, -0.4241288900375366, 0.9187843203544617, -0.5355886220932007, 0.6373490691184998, -0.004209650680422783, 0.45642736554145813, 0.9507192969322205, 0.0012600511545315385, -0.8467272520065308, -1.5092469453811646, -0.7679747343063354, 0.8667184114456177, 0.02818356640636921, -1.2250689268112183, 0.6584675312042236, 0.6667025089263916, -0.637418270111084, -0.21216462552547455, 0.0439770333468914, 0.7140728235244751, -0.4523118734359741, -0.0035214528907090425, -1.3051387071609497, 0.9172087907791138, 2.2380211353302, 0.6892547607421875, 0.9885416030883789, 1.1485811471939087, 0.8428846597671509, -3.349987745285034, 0.03430497273802757, -0.027806932106614113, -0.8624555468559265, 3.0331027507781982, 0.6291213631629944, 0.2433859258890152, 0.4321422576904297, -0.9545331597328186, 1.1895776987075806, -1.5295039415359497, -1.619526743888855, 0.40473005175590515, 0.2762150168418884, 0.22249123454093933, 0.9591137170791626, 0.8405739665031433, 0.19424760341644287, -0.7388560771942139, -0.8720794916152954, -0.36262959241867065, -0.26300930976867676, -0.12518490850925446, -0.6677502393722534, 0.4347057640552521, -0.7338005304336548, -1.2140166759490967, 0.7523046135902405, 0.9092040061950684, 1.1261793375015259, -0.14159496128559113, -2.2714343070983887, 0.41226494312286377, -0.04393656179308891, -0.24677029252052307, 0.43401065468788147, -0.4411400556564331, 0.7054181098937988, -0.6264756917953491, -0.6506987810134888, -0.7987812161445618, 1.9579302072525024, 1.025901436805725, -0.5991092920303345, 0.14291861653327942, -1.8525583744049072, 1.0338598489761353, 1.1269456148147583, -0.7791557312011719, -0.6039847731590271, -1.0271766185760498, 0.1908637285232544, -0.40265554189682007, -0.3630133867263794, 0.3683040142059326, -1.6610324382781982, -0.4802069067955017, -0.22401411831378937, 1.016720175743103, 1.4020887613296509, -0.5561023354530334, 1.0194098949432373, -1.4369868040084839, -1.1427092552185059, -0.48655903339385986, 1.8397221565246582, 0.4878632128238678, 0.5848598480224609, -0.3844841420650482, 0.7547305822372437, 1.109336256980896, 1.3093962669372559, 0.06097065284848213, 0.24940736591815948, 0.03629133105278015, 0.3469569683074951, -0.34756016731262207, -0.4339748024940491, 0.3815506100654602, 0.5420888662338257, 0.4792465269565582, -1.4127477407455444, -2.030338764190674, 0.9573865532875061, -1.3105543851852417, -2.466742753982544, 0.5207438468933105, -1.6340864896774292, -2.4690892696380615, -0.19884400069713593, 0.7490894794464111, -0.5795524716377258, 0.2622803747653961, 1.0707091093063354, 1.4664188623428345, 1.0288090705871582, -0.5747306942939758, 0.6252791881561279, 0.8866420388221741, -0.018547791987657547, 1.3905047178268433, 0.08727233856916428, -0.3173983693122864, -0.18736064434051514, 0.4163314998149872, 1.4270038604736328, 0.7919051647186279, 0.2689814567565918, 0.7631108164787292, 0.01522868312895298, 1.3059948682785034, 0.428972989320755, 1.37272047996521, 0.49674391746520996, -0.050149474292993546, -2.2146799564361572, 0.039515867829322815, -0.59766685962677, -0.25511521100997925, 0.03709550201892853, 0.7477666139602661, -0.2652457058429718, -0.44760310649871826, -0.5662150382995605, -0.4799707531929016, -0.20713335275650024, -0.7012652158737183, -0.14619889855384827, 0.19828489422798157, 0.4814048707485199, 0.2380480170249939, 0.723859429359436, 0.7762088775634766, -1.1206865310668945, 0.700796902179718, 0.8692492246627808, 0.13404716551303864, -0.7510902881622314, 1.2243103981018066, 1.0855424404144287, -0.48598966002464294, -0.37003636360168457, 0.7770342826843262, 2.2566561698913574, 0.7068371772766113, 0.2952311933040619, 0.03263493999838829, 0.3510693907737732, 0.6327549815177917, 0.21558047831058502, -1.3400589227676392, -0.31538987159729004, -1.3169810771942139, 0.4578608274459839, 0.3918217122554779, -0.6833992004394531, 0.30466118454933167, 0.20693719387054443, -1.2115057706832886, -0.14955338835716248, 1.4940755367279053, 0.2379605621099472, 0.6584784984588623, -0.021936167031526566, -0.8288280963897705, -0.7292343378067017, -0.6724364161491394, 0.5357033014297485, -0.8448548316955566, -2.2285103797912598, 0.1921916902065277, -0.9303233623504639, -0.6183716654777527, -2.0098049640655518, 1.2202426195144653, 0.0021394705399870872, 0.5697345733642578, -2.918630361557007, -1.0446743965148926, 0.6358388662338257, 0.425581157207489, 1.5548338890075684, 0.11885621398687363, 0.6314651966094971, -0.12020624428987503, -1.1324262619018555, 0.7770127654075623, -0.7846510410308838, 1.1581205129623413, -1.2895530462265015, 0.920101523399353, 0.9140111804008484, -0.6376311779022217, 0.7027468085289001, 1.9028234481811523, 1.207679271697998, 0.9233298897743225, 0.31476154923439026, 0.6784711480140686, 1.1592533588409424, 1.43625807762146, -0.19808368384838104, 1.5433447360992432, 1.027512788772583, 0.06876921653747559, 0.3569662570953369, -0.0896565318107605, 1.0781527757644653, -0.14462800323963165, -0.7537595629692078, 0.2879394590854645, 0.8619881272315979, -0.24457433819770813, 0.7634536623954773, 0.4775887429714203, 0.5208061337471008, 0.9439992308616638, 1.5097105503082275, -0.3762974441051483, -1.3718301057815552, 0.42730772495269775, -1.967179775238037, 0.4898484945297241, 1.1785178184509277, -0.9253077507019043, 1.0728968381881714, 1.7850929498672485, -1.2972428798675537, -0.5226593613624573, 1.3076064586639404, -0.7805964350700378, -0.27035290002822876, 1.2246874570846558, 0.6946780681610107, -1.3225775957107544, -0.8970884084701538, 0.5255709290504456, 0.3497142791748047, -0.07808732986450195, -0.7853306531906128, -0.1665743738412857, -0.2034350484609604, -1.5447320938110352], [-2.0653464794158936, -0.5078493356704712, -0.021543430164456367, -0.4574739634990692, -0.13907955586910248, 1.6816163063049316, 0.1823282092809677, -0.46539974212646484, 0.13193197548389435, 0.030622484162449837, 1.0088053941726685, 1.3860057592391968, 0.5585854053497314, 0.22259633243083954, 0.8120273947715759, -0.06143468990921974, -0.21443207561969757, 1.5118963718414307, 1.145176887512207, 1.459000825881958, -1.5740349292755127, -0.7010491490364075, -0.7494390606880188, -0.06781339645385742, -0.3676954209804535, -1.772329568862915, -0.7620868682861328, 0.9708663821220398, 1.6853162050247192, 0.04015054553747177, 1.1265674829483032, 0.09533147513866425, -0.37318289279937744, 0.43943503499031067, 0.8566625118255615, -1.7841784954071045, -2.1755146980285645, -1.8942170143127441, -0.11875076591968536, 0.03964736685156822, 1.6707055568695068, -0.8926131725311279, 0.3206443786621094, 0.3581497073173523, -0.11875366419553757, -1.77375066280365, 0.3386162221431732, -0.19504797458648682, 0.8941463828086853, -0.013523499481379986, -0.9256787300109863, 0.2989252805709839, 0.6026911735534668, -0.3316227197647095, -0.7714568972587585, -2.9118549823760986, -0.5344547629356384, 0.821597158908844, -1.126882553100586, 1.530365228652954, -0.9255999326705933, -1.228110671043396, 0.8580915331840515, -0.39675480127334595, -2.0835981369018555, 0.7013905644416809, -0.6771151423454285, 2.085813522338867, 1.190392255783081, 0.4451597034931183, 0.42549365758895874, -0.3727101683616638, -0.7159484028816223, 1.0033485889434814, -0.11996478587388992, -0.8023663759231567, -0.5171735882759094, 0.8230650424957275, -2.1942944526672363, 0.429054319858551, -0.8409968614578247, -0.42364710569381714, -0.30237701535224915, -0.060300279408693314, -1.0817344188690186, 1.3677901029586792, -0.22276373207569122, -0.18616291880607605, 0.9650673866271973, -0.39938101172447205, -0.9201992154121399, -0.3759497404098511, -0.6564046740531921, 0.7440073490142822, -1.2428330183029175, 0.48692312836647034, 0.17944744229316711, -0.195733904838562, 0.17767639458179474, 1.5224816799163818, 0.10905586183071136, 0.5090335607528687, 0.1271696388721466, -0.40517541766166687, 0.16970768570899963, 0.3471885919570923, -0.36661654710769653, -1.774003505706787, 0.9355759620666504, -1.1058826446533203, 0.4705996811389923, -0.7224016785621643, -1.7780646085739136, 0.7151568531990051, -0.7728602290153503, 2.835545301437378, 0.5224178433418274, -1.5433833599090576, 0.44477105140686035, -1.1073908805847168, -1.1189353466033936, 0.3498706519603729, 0.9081386923789978, 0.26454293727874756, -0.22211194038391113, -1.190586805343628, 0.14281664788722992, 1.6798746585845947, -0.11314419656991959, 1.6558645963668823, 1.5413984060287476, -0.5873176455497742, -2.0836596488952637, 0.033661987632513046, -0.12574049830436707, 0.39176079630851746, -0.5580280423164368, 0.7852917909622192, -0.89247065782547, -0.6760573387145996, -2.400897979736328, -0.9462034106254578, 0.1942201852798462, -0.9141971468925476, 1.2358494997024536, 0.4664410650730133, -0.8426806330680847, -0.6630192995071411, -1.8414345979690552, 1.6369134187698364, -0.9311633706092834, -0.7396758794784546, -0.8669630885124207, -0.4293302893638611, -0.07225262373685837, -0.015607211738824844, 1.541472315788269, 1.1692194938659668, 2.01526141166687, -0.6657602787017822, -1.1977880001068115, 0.642206609249115, -0.31053492426872253, 1.104188323020935, -0.5257250070571899, 0.9516257643699646, -0.1720675528049469, -0.9300352931022644, 0.24442632496356964, 0.6066851019859314, 0.09902766346931458, -0.2722594141960144, -1.468996524810791, 0.20833519101142883, -0.05050427094101906, -1.0340486764907837, -1.8052712678909302, -0.9459728598594666, 0.6302597522735596, 0.8368462324142456, -1.1760958433151245, 1.3675450086593628, -0.24859687685966492, -0.2840830385684967, 0.43469932675361633, 0.8432846069335938, -1.6050180196762085, 0.7633246183395386, 1.3540759086608887, 0.8732668161392212, -0.7925826907157898, 0.3297625184059143, 0.5416712760925293, 0.28240442276000977, -0.6760740876197815, -0.04518750682473183, 0.8391828536987305, 0.3339497447013855, 0.613832950592041, 0.6923375129699707, 0.9216569662094116, 1.3880455493927002, -0.27321383357048035, -0.47966957092285156, 1.097082495689392, -1.3692983388900757, -0.5905705094337463, -0.5964481234550476, 0.6288968324661255, -1.1608408689498901, 1.1424555778503418, -0.7095038890838623, 1.4238721132278442, -1.2785608768463135, 1.990945816040039, -1.7327470779418945, -0.29408183693885803, 0.39374661445617676, 1.290998101234436, 1.1414822340011597, 1.0741605758666992, 1.01015043258667, 0.29450732469558716, -0.9098947048187256, -1.2862592935562134, -0.302259624004364, 2.4162521362304688, -0.5195793509483337, -0.2388296127319336, -1.4212026596069336, -0.0009253804455511272, -1.2442189455032349, 0.3016482889652252, 0.2590983510017395, 1.5675485134124756, -0.2100607305765152, 1.2326503992080688, -0.6287932395935059, -0.32865604758262634, -1.0139750242233276, -0.37569478154182434, 0.5137280821800232, -0.7323541641235352, 1.0298432111740112, 0.6294912695884705, -0.7003139853477478, 0.3370714783668518, -0.42734846472740173, 0.5776441097259521, 0.14799420535564423, 0.5781416893005371, -0.3394917845726013, -0.7314162850379944, -0.4953434467315674, 2.9326071739196777, 0.18633873760700226, 0.11180522292852402, 0.012770598754286766, 0.1325693130493164, 1.5075006484985352, 0.03411109372973442, 0.8607689142227173, -1.2577097415924072, -0.3351399302482605, 1.4498875141143799, -0.6049485206604004, -1.8447203636169434, -1.025052785873413, -1.4547648429870605, 0.1134393960237503, 0.6099117994308472, 0.811871349811554, -0.2544722855091095, -0.08061371743679047, -0.09550021588802338, -0.330875039100647, -0.8096218705177307, 0.46192312240600586, -0.03617784380912781, -0.5101554989814758, 0.03231092169880867, 0.4973601996898651, 0.606825053691864, -0.6984943747520447, -0.4192851185798645, 0.7975283265113831, 1.1047890186309814, -0.8010287880897522, 0.9718736410140991, -1.862545132637024, 0.24789007008075714, 0.5775772333145142, 1.228346347808838, 1.1058344841003418, 0.7118802070617676, 0.7842276096343994, -0.2980063557624817, 0.7654051780700684, 0.464643657207489, -0.6751977205276489, 0.18460705876350403, -0.009979517199099064, -1.3944512605667114, -0.9932796359062195, -0.5425156354904175, -0.3612711429595947, 0.12024139612913132, -1.7073737382888794, 1.2070244550704956, 0.7335792779922485, 0.2846086919307709, -1.3706088066101074, 1.0503708124160767, 0.15206734836101532, 0.03445091098546982, 0.33003392815589905, 0.36189374327659607, 0.6613011956214905, 1.591179370880127, -0.35931819677352905, 1.1075469255447388, 1.4985039234161377, 1.0552986860275269, 0.16033418476581573, -0.1530771255493164, -1.767866611480713, -1.0396839380264282, 0.882453203201294, 1.184635043144226, -1.1260719299316406, 0.49869418144226074, -0.076825812458992, 1.4862374067306519, -0.143188938498497, -0.9346045851707458, 1.2308908700942993, 0.7197004556655884, -1.8242963552474976, 2.047285795211792, -1.1905372142791748, 0.9728518128395081, 2.351818323135376, -1.1076252460479736, 1.6161173582077026, -2.047039747238159, -0.7823837399482727, -0.024658748880028725, 0.0642380341887474, -0.48402249813079834, -0.6183056831359863, -0.5287737250328064, -1.7134126424789429, 0.5699191689491272, -0.011901481077075005, -1.2067253589630127, -0.4068969488143921, 0.5053016543388367, 0.4323805272579193, 0.6333110928535461, -1.832473635673523, 0.33696725964546204, 1.3786908388137817, 0.4224911332130432, 0.39690616726875305, 0.5460332036018372, -0.2516355812549591, -0.4634835124015808, -0.8628952503204346, -2.2096312046051025, 0.540804386138916, -1.4952362775802612, 1.9456735849380493, -0.9653483033180237, 1.2679072618484497, 0.8357996344566345, 1.3584706783294678, -0.33275553584098816, -0.9790865182876587, -0.4708975553512573, -2.0627567768096924, -0.8445904850959778, 0.9046303033828735, 1.2548130750656128, -2.906750202178955, 1.7768892049789429, 0.3674795925617218, 0.9730479121208191, 0.2881995141506195, 0.14675943553447723, 1.2341667413711548, -0.021588312461972237, 0.5471675395965576, -1.075879454612732, -0.04151999577879906, -0.3416516184806824, 0.18998725712299347, 1.8184548616409302, -1.138519048690796, 0.46712228655815125, -0.4507209360599518, -0.694225013256073, -0.8429599404335022, -0.962834894657135, -0.1261700987815857, 0.3679785430431366, 0.6621617078781128, -0.7009344100952148, -0.4859183132648468, -0.1904369443655014, -0.5742632746696472, 0.43954378366470337, 0.4058070480823517, 1.9742321968078613, -0.21984055638313293, 0.11739072948694229, -0.5175353288650513, 0.7767316102981567, 0.6961809992790222, -1.5169481039047241, -1.1343398094177246, 0.14922800660133362, 0.7072133421897888, 0.5763685703277588, -0.7842450141906738, 0.5827770829200745, 0.06716238707304001, 0.19881965219974518, -1.762636661529541, 1.2331538200378418, -0.6088464260101318, 0.8897279500961304, 0.9355518817901611, -1.8670461177825928, -2.0343785285949707, -0.43719953298568726, 1.3187333345413208, 0.7522592544555664, 0.14345961809158325, -0.06309489905834198, -1.1899422407150269, 0.3954816162586212, 0.6455597877502441, 0.016218509525060654, 0.34872493147850037, 1.3003429174423218, 0.2704373896121979, 1.431039810180664, -0.9791497588157654, -2.9069576263427734, 0.0007185235153883696, 1.3138529062271118, 0.929326593875885, 0.24402070045471191, -1.0018270015716553, -1.3480160236358643, 1.9876179695129395, -1.0065757036209106, 1.5549507141113281, 0.15488900244235992, 1.423997402191162, -0.5574175715446472, -0.4032805562019348, -0.10429792106151581, 0.8532022833824158, 0.9340351223945618, -2.4955484867095947, -2.2659382820129395, -1.335976481437683, 0.37939453125, -0.38373151421546936, 1.092233657836914, 0.02740456722676754, 2.0930604934692383, -0.9001310467720032, 0.6631699800491333, -2.9988949298858643, 0.21368713676929474, -0.6474527716636658, 0.1652272492647171, -0.044607002288103104, -1.0460742712020874, -1.4716732501983643, -0.14675992727279663, -0.5112719535827637, -0.11524014919996262, 0.41418957710266113, 0.3519339859485626, 1.7563488483428955, 1.6805922985076904, 1.6751242876052856, 0.7128422856330872, 0.27894964814186096, 1.322133183479309, -0.6566179394721985, 0.9768006205558777, -0.6064248085021973, -1.5184154510498047, -0.386050820350647, 0.3851528465747833, 0.1563970446586609, 0.10545109212398529, 0.010123549029231071, 0.9620220065116882, -1.4986635446548462, -1.5602296590805054, -1.1096099615097046, 0.38859179615974426, 0.9490442276000977, -0.6337774991989136, 0.16614679992198944, 0.6631219983100891, -1.060958981513977, 0.8824772238731384, -0.5522485375404358, -1.5097297430038452, 0.10581395775079727, 1.6452689170837402, -0.21684935688972473, -0.2937160134315491, -0.1664493829011917, 0.2986469566822052, 0.03763476759195328, -0.6749743819236755, -1.9428269863128662, -0.814552903175354, 1.1503818035125732, -0.664409339427948, 0.550646960735321, 0.7701575756072998, 0.9583913683891296, 1.0419931411743164, 0.3796221911907196, 1.7964106798171997, 0.7824499607086182, -1.2291336059570312, -1.897183895111084, -0.31931647658348083, 0.605858325958252, 0.8478543758392334, 0.7624837756156921, -1.4365522861480713, 0.6720215678215027, 2.0342071056365967, 0.45281022787094116, -1.2175976037979126, 0.04354490712285042, -1.5021799802780151, -1.7952423095703125, -1.0656285285949707, -0.3323691189289093, -0.42573127150535583, -0.09727513045072556, 0.036848001182079315, -0.2024274319410324, 0.6873237490653992, -1.6580764055252075, -1.3308318853378296, -0.09083963930606842, 0.38261932134628296, -0.6557612419128418, 1.5934265851974487, -0.785503089427948, 1.9486716985702515, 1.370624303817749, 1.1508779525756836, 1.7284796237945557, 0.4813951551914215, 0.14055517315864563, 0.22545987367630005, -0.9365798234939575, -0.9824833273887634, -2.034837007522583, -0.5144225358963013, -1.347944736480713, 0.7577582001686096, 0.1207456886768341, -0.6758792996406555, -0.610942542552948, -0.45953568816185, 1.1781338453292847, -0.3640595078468323, -0.8323787450790405, -0.943981945514679, -1.1564512252807617, 0.4802566468715668, -0.1782507598400116, -0.7096403241157532, 2.038663864135742, -0.5829755067825317, 0.7658827304840088, 0.804624617099762, 0.20336417853832245, -0.7352712154388428, -0.8724176287651062, 1.2448210716247559, -1.1971991062164307, -0.1238052174448967, 0.5589604377746582, 0.24999511241912842, 1.8423172235488892, -1.809832215309143, -0.07970067858695984, -1.4803736209869385, 1.0414096117019653, 1.486703634262085, -0.011832302436232567, -0.4672766327857971, 0.8931264877319336, -0.18471018970012665, -0.6379874348640442, 0.3533608913421631, -0.34351277351379395, -0.5026688575744629, -0.6972234845161438, 0.6756529211997986, -0.10850955545902252, -1.2636147737503052, 0.1807384490966797, -0.27634087204933167, 0.20499461889266968, -1.7922958135604858, -1.7315536737442017, 0.831041157245636, 0.22315841913223267, 0.8219269514083862, 0.07167781889438629, -0.34648972749710083, 0.4135035574436188, 0.17562825977802277, -0.13024720549583435, 0.32901641726493835, -0.5684597492218018, 0.11037836223840714, 0.26104190945625305, -0.620462954044342, -0.5252890586853027, 0.793237030506134, -0.4531841576099396, -0.027605686336755753, -1.4089587926864624, 2.5564513206481934, 1.9845114946365356, 0.8784475326538086, 0.776583731174469, 0.08396276831626892, -3.5574073791503906, -0.3193379342556, -0.38926640152931213, -2.7383792400360107, -0.5254303216934204, -0.5402491092681885, 0.4053122103214264, -0.6523393988609314, 0.49562472105026245, 0.9904269576072693, 0.26697540283203125, 0.6229668259620667, -1.5262247323989868, -1.1199697256088257, -1.5114936828613281, -0.875795304775238, 2.2785990238189697, -1.2100234031677246, 0.6873255968093872, 0.008149449713528156, 0.3193274140357971, -0.6985648274421692, -0.718612551689148, 0.851934015750885, -0.9550594687461853, -0.460623174905777, -2.568511486053467, 0.4507494270801544, 1.825081706047058, 1.261904001235962, 1.1698074340820312, -0.8294995427131653, 0.5668860077857971, -1.9647347927093506, 0.8071433305740356, -0.40111708641052246, 1.6609258651733398, -1.002489686012268, 1.8292651176452637, -1.1765096187591553, -0.4270898699760437, -0.1558235138654709, -0.27062851190567017, 0.5786662697792053, 1.154954433441162, 0.03362654522061348, 0.744570255279541, -0.9110049605369568, 1.4262841939926147, -0.2483942210674286, -0.28115636110305786, -0.9482376575469971, 1.0310065746307373, 0.1696830838918686, 0.21580497920513153, -0.3311309516429901, 0.7231190204620361, -0.25968247652053833, -1.8488527536392212, 1.4954755306243896, -0.5236618518829346, 0.7109881639480591, 0.41598305106163025, -0.24497827887535095, -0.8925142288208008, 0.7765781879425049, 0.06751983612775803, -0.6849499344825745, 0.6635984778404236, 1.953687071800232, -0.10643299669027328, -1.4698095321655273, -1.5190622806549072, -1.2059390544891357, 0.260606974363327, -1.3436986207962036, -0.5724984407424927, 0.5482193827629089, 2.3250327110290527, 0.494957834482193, -1.1709637641906738, 0.427251935005188, -0.9490695595741272, -0.1444326937198639, -0.09428418427705765, 1.1210743188858032, -0.27411526441574097, -0.2702445685863495, 0.8769628405570984, 1.0341964960098267, -0.5956553220748901, 1.451882243156433, 1.5531052350997925, -1.6657843589782715, -0.9551755785942078, -0.45027151703834534, -0.3776156008243561, 0.2446533441543579, -0.6627861261367798, 0.8558622598648071, 0.2850572168827057, 1.480785608291626, 0.6152472496032715, 1.237359881401062, -0.12078717350959778, 1.553354263305664, -0.7894766330718994, 1.3068877458572388, -0.499103307723999, 0.825699508190155, -0.7067870497703552, 0.6184641122817993, 0.22497761249542236, -0.6439894437789917, 0.6261403560638428, 0.016619209200143814, 0.12871897220611572, 1.154030680656433, -0.45488572120666504, 1.2150334119796753, 0.0651240274310112, -0.789789080619812, -0.9625363945960999, 0.17128664255142212, 0.979385256767273, 0.10224483162164688, 0.6524399518966675, 0.8033877611160278, 0.5566849708557129, 2.334735870361328, -0.8789539337158203, -1.753321647644043, -1.1439508199691772, 0.17059452831745148, 2.8397116661071777, -1.1368422508239746, -1.5377264022827148, -1.3545271158218384, -0.7499687075614929, 0.1653243750333786, 2.01116943359375, -1.0162137746810913, -0.820992112159729, -0.18783454596996307, 1.4864062070846558, -1.6587194204330444, 1.2453604936599731, 0.8660228848457336, 0.25233739614486694, 1.0721406936645508, 0.896816611289978, -1.279001235961914, -0.19522517919540405, -0.947909951210022, -1.621262788772583, -2.034416913986206, -1.132999300956726, 0.9806581139564514, -0.13300301134586334, 1.051641583442688, -1.7537798881530762, 2.4958527088165283, 0.5102991461753845, -0.6277933120727539, 2.0897984504699707, 0.7930389642715454, -0.4237734079360962, -0.49466919898986816, -0.8849117755889893, -2.215179443359375, -0.01878225803375244, 1.1639395952224731, -0.3412677049636841, -0.9972706437110901, -2.205836534500122, 0.16090334951877594, -0.3954327702522278, 1.1174681186676025, -0.3648357093334198, -0.8063091039657593, -0.9633597135543823, 0.921073853969574, 0.3032267391681671, -0.724986732006073, 1.5382617712020874, -0.4243578016757965, 1.3099631071090698, 0.5639680624008179, 1.345810055732727, -0.405488520860672, 0.5938661694526672, -0.6051780581474304, -0.1584010124206543, 0.8363637328147888, 0.6689297556877136, -0.9966307282447815, 0.6578237414360046, -0.2531604766845703, -0.14183999598026276, -0.4581485390663147, -0.5781486630439758, 0.9017193913459778, -1.2100145816802979, -1.5855892896652222, 1.668696403503418, -0.37563133239746094, 1.3366148471832275, 1.2597384452819824, 0.9152301549911499, -0.3730369806289673, -0.4465789794921875, 1.0248099565505981, -0.1506332904100418, 1.0315324068069458, -1.4586416482925415, -1.213083267211914, -0.44732946157455444, -0.2745159864425659, 1.0500755310058594, 2.2078397274017334, -1.7478803396224976, 0.9738774299621582, 1.2862838506698608, 0.07372497767210007, -0.23348373174667358, -0.7820596694946289, 0.7624549865722656, -0.06274193525314331, 0.5714691877365112, 2.166234016418457, -1.1666529178619385, -0.01272107195109129, -0.7183972597122192, -0.06554550677537918, 1.9065886735916138, 0.3005701005458832, 0.8562373518943787, -1.2452819347381592, 0.5851355195045471, 0.2867927551269531, -1.475429892539978, -0.47972428798675537, -0.7578815817832947, 0.6915424466133118, 0.7628348469734192, -1.1619139909744263, 0.5279637575149536, 0.13731440901756287, 0.15787848830223083, -1.3646587133407593, 1.827216625213623, 0.67724609375, -0.6224267482757568, -1.7418893575668335, 1.1873670816421509, 0.5791226625442505, -1.6166667938232422, 0.17151284217834473, -1.1271092891693115, 0.8927642107009888, 0.25520971417427063, 0.6036121249198914, 0.68561190366745, -0.498041033744812, 0.16250944137573242, 1.2395038604736328, -1.177771806716919, 1.0598613023757935, 0.7134087681770325, -0.27182862162590027, -0.25224751234054565, -1.9651806354522705, -0.0169612355530262, -3.220658779144287, 0.8206161856651306, -1.0296963453292847, -0.04717725142836571, 0.9514700770378113, 0.021583745256066322, -2.247020721435547, -1.523977518081665, 1.3356925249099731, -1.3609421253204346, -0.4435335397720337, -0.9383007884025574, -0.4984453022480011, -0.8574283123016357, 0.5229393243789673, -0.2612258493900299, 0.8803120255470276, 0.6131442785263062, -0.6885846853256226, 0.6981694102287292, 2.0387282371520996, 0.7653182148933411, -0.9684248566627502, 0.4960266947746277, 0.1239025667309761, 0.7908840179443359, -1.3327969312667847, 1.1415867805480957, 0.16367876529693604, 0.7791945338249207, 0.17833374440670013, -0.19062256813049316, 0.7009709477424622, 0.8329660296440125, -1.3235734701156616, -0.16810008883476257, -1.22662353515625, 0.18626610934734344, -0.12752017378807068, -0.04972084239125252, -2.090388059616089, -0.707198441028595, -0.42931604385375977, -0.11162898689508438, 1.4248669147491455, 0.19001032412052155, -1.6248915195465088, 0.6334577798843384, 0.640112042427063, 1.1063185930252075, -1.959024429321289, 1.2692499160766602, 0.29113608598709106, -0.5144116282463074, -2.110888957977295, -0.01734231226146221, -0.5348945260047913, -0.08610022813081741, 1.249147653579712, 0.08619164675474167, -0.46205195784568787, 0.1930657923221588, 0.5323254466056824, -1.051551342010498, 0.8183291554450989, -0.17518749833106995, 1.0934535264968872, 1.0904027223587036, 0.9175750613212585, 0.8385670185089111, 1.8863723278045654, 1.8274518251419067, -0.07273667305707932, -1.4164470434188843, -1.35965096950531, 0.27326345443725586, 0.44834983348846436, 0.09898115694522858, -1.384169578552246, 1.0821698904037476, -1.0334477424621582, 0.21333523094654083, 0.6787083745002747, -0.6165167689323425, 1.1623718738555908, -0.25378042459487915, 1.2847380638122559, 1.4169440269470215, 0.48691269755363464, 0.6288685202598572, -0.200392946600914, -0.09517408162355423, -1.2661224603652954, 0.0006954957498237491, -1.1296372413635254, 0.4398514926433563, -1.9751516580581665, -0.9743168354034424, 0.4877341091632843, -0.6789118051528931, -0.5528327822685242, -0.9716365933418274, 0.25088903307914734, 0.1504947692155838, -0.9695321917533875, 1.124036431312561], [0.8823620080947876, -1.0744004249572754, 0.014702717773616314, -0.44851788878440857, 2.449279308319092, 0.4125783443450928, 1.9063211679458618, 1.880266547203064, -0.8227802515029907, 0.020526496693491936, -1.1267707347869873, -1.219037652015686, 1.1231639385223389, -0.36730170249938965, 0.8776358962059021, 0.30979374051094055, 1.458773136138916, -1.762290358543396, 1.1176844835281372, -0.855615496635437, -0.14655077457427979, -0.5717348456382751, -0.4443417489528656, -1.1588072776794434, 0.5933136940002441, -0.3210740089416504, -1.8901362419128418, 0.14803573489189148, -0.9570105671882629, 0.8275023102760315, -0.7050809860229492, 0.1514936089515686, -0.1422639787197113, -1.2239562273025513, 1.1446131467819214, 0.1395941525697708, -0.742505669593811, -0.4367419481277466, -2.1078360080718994, 1.4772279262542725, 0.33586591482162476, 0.41114121675491333, -1.8529433012008667, -0.6614225506782532, 1.1303372383117676, -0.5119790434837341, -0.44018253684043884, -1.4987123012542725, 0.3498576283454895, 0.4606544077396393, -0.3837863504886627, -0.27733513712882996, -0.4400577247142792, -1.2838096618652344, 0.8077905774116516, 0.5785590410232544, -0.05271364003419876, -0.33689233660697937, -0.06130196899175644, 0.38212695717811584, 1.768595576286316, 0.30939415097236633, -0.1369868367910385, -1.9993891716003418, -2.2300615310668945, -1.3250010013580322, 1.4134918451309204, 0.2709352374076843, -0.4915180802345276, 0.4328317642211914, -0.2260269820690155, -0.14768007397651672, -0.984287679195404, -1.544295310974121, -0.04146338999271393, -1.5932306051254272, -0.39075666666030884, 0.6036041378974915, 0.14663933217525482, 0.8858249187469482, -1.5326684713363647, -0.3647467792034149, -0.8750024437904358, -0.29423612356185913, -0.36388707160949707, 0.8621308207511902, 1.8386708498001099, -1.3951451778411865, 0.08444231748580933, 2.654186487197876, -2.498727321624756, -0.5651509165763855, -1.620416283607483, -0.7330523133277893, 0.6815706491470337, 1.1531145572662354, 1.4001034498214722, -1.7938244342803955, -1.9435936212539673, -0.9662742018699646, -0.9932059645652771, -0.28453969955444336, -0.69486403465271, 0.22709135711193085, 1.038023829460144, -1.49920654296875, 0.33756622672080994, -0.6945315003395081, 0.6339439153671265, -0.8787254095077515, 0.1756085902452469, 0.24662461876869202, -0.05588241666555405, -1.639691710472107, 1.4171901941299438, 0.5901476144790649, 0.06771921366453171, -0.2518187463283539, -1.579415202140808, -1.6025640964508057, -0.2886925935745239, -0.6691349744796753, -0.7427933216094971, 0.4392107129096985, -0.17765004932880402, -0.07676994055509567, 1.1174135208129883, -0.330557256937027, 1.5049997568130493, -1.3590896129608154, -1.6138876676559448, 1.3622300624847412, 1.3335281610488892, -1.4803684949874878, -0.546344518661499, -0.5097987055778503, 0.4143243730068207, 1.3492203950881958, 0.7235341668128967, -0.37360405921936035, 0.14663982391357422, -1.653267741203308, -0.574099063873291, -0.3432560861110687, 1.1070201396942139, -2.289766788482666, -2.1457934379577637, 0.0039062597788870335, 1.2688990831375122, -1.556995153427124, -0.8818712830543518, 0.9285731911659241, -0.16147549450397491, 0.3338264226913452, 1.4294883012771606, -0.2590640187263489, 0.988825261592865, 0.06038179621100426, -0.5481351017951965, 1.4555608034133911, 1.0536673069000244, 0.4488399624824524, 1.459945797920227, 0.583258330821991, -0.06522803008556366, -0.711130678653717, 0.9855818152427673, 1.0485743284225464, -0.06722808629274368, -0.6380094289779663, 0.7314149737358093, 0.20255710184574127, 1.1745798587799072, -0.8440468907356262, 0.5980573296546936, -0.8952662944793701, 0.7409624457359314, 1.5410667657852173, 1.0557637214660645, 0.6917350888252258, -2.117724657058716, -0.6147223114967346, -0.12277424335479736, -0.637598991394043, -2.133983612060547, -1.0760834217071533, -0.04773169755935669, 0.12895183265209198, -0.6145651936531067, 1.5937168598175049, 0.03147147595882416, -0.5378881096839905, 0.7118949294090271, -0.8397011756896973, -0.3354787230491638, 0.5795408487319946, -0.6295405626296997, -1.1758148670196533, 1.3795053958892822, 1.4264564514160156, -1.5062836408615112, 3.4924869537353516, -1.4509544372558594, 0.0008570097852498293, 0.8514661192893982, -1.3457574844360352, -1.0132967233657837, 0.059908878058195114, -0.9764497876167297, 0.662654459476471, -0.8878424167633057, 0.5775786638259888, -0.965976357460022, -0.2995177209377289, -1.6006579399108887, 1.4125624895095825, -1.5353485345840454, 1.2029430866241455, 0.119169682264328, -0.3473764955997467, 0.876777172088623, 0.25279098749160767, 1.0247927904129028, 0.09086418151855469, 0.17158450186252594, 0.12923181056976318, -1.2212669849395752, 1.8726361989974976, 0.9886850118637085, -0.9879478812217712, -1.2525120973587036, 1.615772008895874, 1.146712064743042, 0.5091261267662048, 0.7967302799224854, -0.33969759941101074, 0.5304549932479858, -0.9428839683532715, -1.083907961845398, -0.2839447855949402, -0.12447100132703781, 0.06032693013548851, -0.09956502914428711, 1.296910047531128, 0.2925812304019928, 0.1929088532924652, 0.35185864567756653, -1.3651891946792603, 0.035398196429014206, -0.01653951033949852, -0.2926059067249298, 0.05912677198648453, 0.778750479221344, 1.3169232606887817, -0.06371898204088211, -1.5715938806533813, 0.32026207447052, -0.5985944271087646, 1.8002028465270996, -0.12297973036766052, 1.2185271978378296, -1.3535431623458862, -0.38074493408203125, -0.01909305714070797, -0.8050677180290222, 0.8623578548431396, 0.9925158023834229, 0.30524009466171265, 1.0382963418960571, -1.7557036876678467, 0.2968664765357971, 0.009674357250332832, -0.04077077656984329, 0.9972463846206665, 0.41613638401031494, 0.8979387283325195, -0.11964854598045349, 0.8101394772529602, -0.8693366050720215, -0.25145211815834045, -0.082276850938797, 0.4622754454612732, -0.5866045355796814, 0.551666796207428, 0.520796537399292, -1.0293149948120117, -0.006282314658164978, -1.8343477249145508, 0.43612170219421387, -0.21325400471687317, 0.3078579902648926, -1.663360834121704, -0.307140588760376, 0.428111732006073, 0.44322139024734497, 3.275754451751709, 0.0730060264468193, -0.10658010840415955, -0.33399203419685364, 0.7153941988945007, 0.03611423075199127, -0.842536211013794, -1.422568440437317, -0.2397381216287613, -0.3771003484725952, 0.5688012838363647, 0.7851316928863525, 0.18881060183048248, 0.9535529613494873, 1.3462886810302734, -0.10148342698812485, -0.02157488651573658, 2.59106707572937, 0.3709971010684967, -0.8246551752090454, -0.9344884753227234, 1.1417133808135986, -0.5507186651229858, 2.1062064170837402, -0.6447227001190186, 0.4055660367012024, 1.0731457471847534, 0.06498298794031143, -0.5726276636123657, 0.07352149486541748, -0.5359006524085999, -0.3467797636985779, 2.9436092376708984, 0.5543528199195862, -1.099571943283081, -0.44645699858665466, -0.05750635638833046, -0.2531947195529938, 0.8230375647544861, 0.03133641183376312, 0.09713704884052277, 1.0295228958129883, -0.25086256861686707, 1.3151129484176636, -0.31613439321517944, 0.5842669010162354, -0.6933099031448364, -0.0236983485519886, -1.1308934688568115, -0.8105794787406921, 2.080537796020508, -1.7115939855575562, 1.4200178384780884, 0.24109289050102234, 0.7860832810401917, -1.1876777410507202, 0.054407261312007904, 0.5966423749923706, 1.697361946105957, -0.26430872082710266, 1.1010645627975464, 0.9037868976593018, 0.6139369010925293, -1.1492323875427246, 1.811654806137085, -0.8562374114990234, -1.1190673112869263, -1.5689449310302734, -0.1615247279405594, -0.5396074652671814, 1.0970444679260254, -0.8028456568717957, 0.8797637224197388, 0.6702293157577515, -0.2399643361568451, 0.8699570298194885, 0.9346099495887756, -0.9251137375831604, -0.8336520791053772, 1.1201210021972656, -1.1620815992355347, -2.852783203125, -1.4699702262878418, 0.6826508045196533, 0.31749337911605835, 0.6078482270240784, -0.11591209471225739, -0.879508912563324, -1.7125635147094727, -1.4229432344436646, -0.1428300440311432, -0.7550228238105774, -0.08539716899394989, -1.0123794078826904, -1.3593320846557617, -0.2745635509490967, -0.0926390215754509, -1.2757322788238525, 0.032282911241054535, 1.0398328304290771, -0.07168607413768768, 0.20090319216251373, -0.03717803210020065, -0.49349769949913025, 0.6228973269462585, -0.36679360270500183, 0.2947085201740265, -0.7229401469230652, 2.3536839485168457, -0.12208469957113266, -1.0027636289596558, 1.067171573638916, 1.0844237804412842, -0.10599134862422943, 0.16373492777347565, -1.1747074127197266, 0.5606842637062073, 0.36540916562080383, -0.02586858533322811, -0.5327106714248657, 1.8581327199935913, -0.28096213936805725, 0.17059263586997986, -1.4123375415802002, 0.9457590579986572, -0.7521178126335144, 2.4761366844177246, 0.23874393105506897, 0.16357256472110748, -0.4619539976119995, 1.0726851224899292, 0.5912789106369019, 0.33501553535461426, -0.0036835367791354656, -0.1786658614873886, -0.03477676957845688, -0.6673876643180847, -2.307605504989624, 0.4717029929161072, 1.0394163131713867, 0.3378349244594574, -0.1266857385635376, -0.5896953344345093, -0.3909246623516083, -0.686055600643158, 0.3927731215953827, 0.0246441513299942, 0.2775695323944092, 0.9155365824699402, 1.484828233718872, -2.0476865768432617, 0.6248119473457336, 2.9023983478546143, -0.6926352977752686, -0.02229824662208557, 0.06772151589393616, -0.15787868201732635, -0.7571544051170349, 1.3153740167617798, 0.3159692883491516, -1.7007180452346802, 0.051745492964982986, -1.251395344734192, 0.3073534071445465, 0.7783566117286682, 0.294793039560318, -0.2760482132434845, 1.8979225158691406, -0.16821031272411346, 0.8713223338127136, -0.30971261858940125, 1.4505599737167358, 1.033892273902893, -2.7028918266296387, -0.24180443584918976, -0.39546069502830505, 0.04151609167456627, 0.7262281179428101, 0.4432659149169922, 0.6839452385902405, -0.7625490427017212, -0.05799560248851776, 2.2339816093444824, -0.3197300136089325, -0.09339486062526703, 1.5517233610153198, -0.30729037523269653, -1.938917636871338, 0.6739373803138733, -1.715635061264038, 0.2139727920293808, -0.493161678314209, -0.814281165599823, 0.3135291039943695, -0.4703807234764099, 0.0499233677983284, -1.2540171146392822, 0.19552381336688995, -0.3495246469974518, 1.6504230499267578, 1.4614390134811401, 2.2249913215637207, -1.3929901123046875, -2.0119338035583496, -2.863318681716919, -0.3856409192085266, -1.5876836776733398, 1.4117398262023926, -1.214090347290039, 0.18339821696281433, -0.5388573408126831, -0.07338683307170868, 0.9179960489273071, 0.19902022182941437, 0.7094452977180481, -0.8941220045089722, -2.156813144683838, 0.05467074736952782, -0.1674334853887558, 0.5452534556388855, 1.370285987854004, 1.0984337329864502, -0.28839826583862305, -0.27286896109580994, -0.1695520579814911, 0.4611032009124756, 0.7128595113754272, -0.5117372274398804, -1.2630144357681274, -0.10914113372564316, 1.53096342086792, 0.8317083716392517, -0.86640465259552, -0.6696585416793823, -0.9847783446311951, -0.7100062370300293, -0.7954897284507751, 0.4358464777469635, -0.00581824965775013, -0.2755747139453888, -0.1330682933330536, 1.767381191253662, -0.34482574462890625, -0.4014492630958557, -1.436141848564148, -1.5159060955047607, 1.1336216926574707, 0.10700871795415878, 1.567345142364502, 1.2231600284576416, 1.4970495700836182, -0.6273744106292725, 1.0226842164993286, 0.598108172416687, 0.32780778408050537, 0.07954784482717514, 0.3727700710296631, -0.764398992061615, -1.6462233066558838, -1.0988407135009766, 1.7452163696289062, 0.9546607732772827, -0.16767427325248718, 2.6543996334075928, 0.3563115894794464, 0.8027693629264832, 0.1536051481962204, 1.5865310430526733, -1.5736783742904663, 1.4239704608917236, -1.082816243171692, 0.26071932911872864, -1.930477499961853, -0.9497573375701904, 0.6410558819770813, 1.000697374343872, 0.506023108959198, 0.6747552752494812, 1.0486938953399658, 1.7514182329177856, 1.6243866682052612, 1.2713547945022583, -0.6734132766723633, -0.9158217310905457, -0.8320720195770264, -0.7490090131759644, 0.11975758522748947, -0.9620601534843445, 1.0072152614593506, 0.8492907881736755, -0.16133590042591095, -0.6812381148338318, 1.4074921607971191, -0.28755810856819153, 1.765381932258606, 2.129667282104492, 0.11990964412689209, 0.7235667705535889, -2.0097997188568115, 1.067754864692688, 1.5614373683929443, -1.358109951019287, 0.12509846687316895, -0.3160465955734253, -1.0427653789520264, -0.6134140491485596, -0.6231421828269958, 0.7898342609405518, 1.3712472915649414, -0.928482711315155, -0.8047332763671875, -1.2432215213775635, 0.44289088249206543, 0.10218723118305206, -0.36770960688591003, 1.9641164541244507, -2.5760247707366943, -1.317111611366272, 0.27321988344192505, 0.22262762486934662, -1.261842131614685, 0.44981133937835693, -1.2487558126449585, -0.0050245532765984535, 0.5989765524864197, 0.37842780351638794, -0.10760560631752014, -0.006368732079863548, -0.5985580086708069, -0.023968743160367012, 0.5069651007652283, 0.948038637638092, 1.111025333404541, -0.0038224076852202415, 1.7775514125823975, -0.20379070937633514, 0.3198011815547943, 0.5852818489074707, -1.5821194648742676, 0.7649762630462646, -1.4834065437316895, -1.2753310203552246, -1.1418747901916504, -0.9827112555503845, 1.22710120677948, 0.30296802520751953, 1.603976845741272, 1.0828728675842285, -0.2484673261642456, 0.9063818454742432, 0.22052758932113647, 0.06853434443473816, 0.826179027557373, -0.05187986418604851, 0.948047935962677, 1.6011021137237549, -1.2633494138717651, -1.9461710453033447, 0.8834995031356812, 0.44315990805625916, 2.458552837371826, 0.5930780172348022, 0.7814602851867676, 0.5225757360458374, -1.1133310794830322, -0.23955023288726807, -0.8347111344337463, 0.8334589600563049, -0.5813040137290955, 0.8633773922920227, 1.2375108003616333, 0.683505654335022, -0.19975340366363525, 0.5043396353721619, 0.780685544013977, 0.02594815194606781, 0.26263538002967834, 0.5349225401878357, 1.9150992631912231, -0.5706918835639954, 0.006261054426431656, 1.0852816104888916, -0.8554081916809082, -0.7687107920646667, 0.2178221493959427, -0.40254008769989014, -1.5562467575073242, -1.2042715549468994, -1.4552983045578003, -0.6727110743522644, -0.08666335791349411, -0.8316460847854614, -0.4702635407447815, -0.9656432271003723, 0.1471879929304123, 0.6729682683944702, 1.5449059009552002, -0.3876269459724426, -1.335655689239502, 1.1688355207443237, -0.5236923098564148, 1.1334713697433472, -3.024732828140259, -0.49653851985931396, -0.8768747448921204, 0.005576721392571926, -1.139651894569397, -1.1511849164962769, 0.05156010389328003, -0.13364680111408234, -0.9694164991378784, -0.41063690185546875, 0.7796130180358887, -2.456627130508423, -1.8064504861831665, -0.7641997337341309, 1.8743749856948853, 0.0604364350438118, -0.6307889223098755, 2.4450671672821045, 0.8810814023017883, 0.32723376154899597, -0.46052441000938416, -1.8875477313995361, -0.6373494863510132, -1.1504367589950562, -0.30708858370780945, 0.1731616109609604, -0.9828911423683167, -0.7993117570877075, 0.5195193886756897, 0.020974844694137573, -0.8310948014259338, -0.15492983162403107, -1.8797699213027954, 0.22372180223464966, -0.5985747575759888, 0.9143792390823364, 2.2878663539886475, -1.121100664138794, -1.2531015872955322, -0.3069533705711365, 2.00083065032959, -0.7725359797477722, -0.8520224690437317, -1.5135173797607422, 0.39208197593688965, -1.2119868993759155, -1.0259294509887695, 1.0873582363128662, -0.8629056811332703, 0.3815232813358307, -1.2385814189910889, -0.7730404734611511, -1.253638505935669, 1.6758354902267456, -1.6738892793655396, 0.6754356026649475, 0.4408150315284729, -0.7527043223381042, 0.01650829240679741, 0.5159045457839966, 0.8624787330627441, 1.5387238264083862, 0.2534669041633606, -0.8951778411865234, 0.6553122401237488, 0.01228277012705803, 0.826882004737854, 0.3869003653526306, -0.2595472037792206, 0.08265529572963715, 0.3153957724571228, 1.4223167896270752, -0.0806722342967987, 0.2887308895587921, 1.360053300857544, -0.2048833817243576, 1.2767671346664429, 0.9244214296340942, 0.07697117328643799, 0.0500403456389904, -1.4592186212539673, 1.0344873666763306, -1.6456953287124634, 0.07905690371990204, -0.7094249725341797, -1.3455607891082764, -1.524177074432373, 0.7849921584129333, 0.2524466812610626, 0.008510511368513107, 1.3522250652313232, -0.41077473759651184, 0.9140560030937195, -0.33165550231933594, -0.396456241607666, -0.06839888542890549, -1.5164321660995483, -0.6701104044914246, -1.6458885669708252, 0.18321743607521057, 0.5603069067001343, 1.1302502155303955, 0.5487001538276672, -1.9947634935379028, -1.5096622705459595, -1.7110356092453003, -0.06189532205462456, 0.0036713783629238605, -0.0557713583111763, -0.07415297627449036, 0.0812547579407692, -0.17853495478630066, 0.38107922673225403, 0.5391900539398193, -1.1108250617980957, -0.11577031761407852, 0.9968405365943909, -0.8168785572052002, 2.3908677101135254, 0.5587534308433533, -1.4068851470947266, -1.047613263130188, 0.4277140200138092, 0.6982812285423279, -0.961654543876648, -1.028133511543274, 0.8532589077949524, 0.8064841628074646, 0.5922175049781799, -0.6791824698448181, 0.09192436188459396, 0.5027036070823669, 1.269279956817627, -0.3943682312965393, 1.0822962522506714, -0.9439395070075989, -2.0379865169525146, -1.4745615720748901, 0.628923773765564, 0.45961397886276245, -0.8409554362297058, -0.7007635235786438, 4.717199325561523, 0.3237650692462921, 2.0813376903533936, 1.4379218816757202, -0.34516453742980957, -1.330621600151062, -1.0100579261779785, -0.28888648748397827, 0.9031086564064026, 2.7060134410858154, -0.5398534536361694, -0.26038214564323425, -0.020420368760824203, 1.2427639961242676, 0.6093918085098267, -1.7592406272888184, 1.3006048202514648, 0.4216318726539612, -0.8270867466926575, 0.2351830154657364, 1.3776363134384155, 0.24497967958450317, -2.131063222885132, 0.975063681602478, -0.6914396286010742, -1.4491209983825684, -0.152939110994339, -1.4666264057159424, 0.23604921996593475, 2.510773181915283, -0.8180480599403381, -1.4507808685302734, 0.8962944746017456, -0.20817628502845764, -2.669680118560791, -0.7835928201675415, -1.0194896459579468, 0.8576750755310059, -0.2699306011199951, 0.256117045879364, 0.36970844864845276, 0.029112432152032852, 0.327907532453537, 0.06355280429124832, -1.2787836790084839, -0.17882117629051208, 0.008968206122517586, -0.3039568066596985, 1.1026227474212646, 0.4693634510040283, -0.18868674337863922, -0.45337560772895813, 0.460277795791626, 1.0762712955474854, -0.4130321145057678, -0.06779900193214417, 0.6867479681968689, 0.5529392957687378, 0.08678169548511505, -0.9675157070159912, 1.6234164237976074, 0.8675294518470764, -1.8760038614273071, -0.7167031168937683, -0.24033121764659882, -1.0101773738861084, -0.42943671345710754, -0.7305592894554138, 1.1488090753555298, -1.076446771621704, 0.714945912361145, -0.2439989447593689, -1.9732179641723633, 0.20654411613941193, -0.8643710613250732, -0.02297157049179077, 0.738079845905304, -0.8174057602882385, 0.8462777137756348, 0.5289896726608276, -0.8519836068153381, -0.463815301656723, -1.948313593864441, 0.32457196712493896, 1.1007730960845947, 0.7651776671409607, -0.2209632396697998, 0.17033952474594116, 1.1991384029388428, -1.4887144565582275, -1.2210241556167603, 2.8212506771087646, 0.38657256960868835, -0.8364265561103821, 0.2509844899177551, 1.6888389587402344, -0.20807164907455444, 0.3225214183330536, -0.2101561427116394, -1.9125787019729614, 1.3347063064575195, 0.5209534168243408, -1.3844356536865234, 1.2282421588897705, 1.5980110168457031, -1.1657109260559082, -0.16317757964134216, 1.656109094619751, -0.05076718330383301, -0.9370999336242676, 0.04779757186770439, 0.12916803359985352, 0.7017517685890198, -0.971166729927063, 0.5974249839782715, -0.30030688643455505, 0.14532385766506195, -0.04501449316740036, -0.31140366196632385, -0.48440054059028625, 0.7596085071563721, -0.9143149852752686, 1.1718693971633911, 0.6187882423400879, 0.8382967114448547, 1.3350093364715576, 0.4280373454093933, 1.488804817199707, 0.9226163029670715, 0.6317091584205627, -0.6690401434898376, 0.5331361293792725, 1.657195806503296, -0.07746017724275589, -0.6766793131828308, 1.285828948020935, 0.8047977089881897, -0.7632203698158264, 0.48732826113700867, 1.317922592163086, 1.3686368465423584, -0.6381535530090332, -0.9661297798156738, 1.15022873878479, -0.6397522687911987, -0.8131546974182129, 0.6091088056564331, -0.5879048705101013, -0.7110008001327515, -0.5496072173118591, -0.8450109958648682, -1.005340576171875, -0.03452037647366524, 0.04577125981450081, 0.44513994455337524, -0.04434052109718323, -1.2063950300216675, 0.3160085380077362, 2.5592896938323975, -1.1369332075119019, 0.6931399703025818, -0.9167279005050659, 0.35691478848457336, -0.38251885771751404, 2.2699179649353027, -0.46924564242362976, -0.36203742027282715, -0.5415739417076111, -0.6780761480331421, 1.0040854215621948, -1.2476017475128174, 0.503837525844574, -0.7666492462158203, -2.1000208854675293, 0.5998110175132751, -1.2914327383041382, -0.7219135761260986, 1.9208190441131592, 1.4108994007110596, -1.1651493310928345, -2.2256217002868652, 0.20778988301753998, 1.1614934206008911, 0.4403519034385681, 0.3264431357383728], [-0.597839891910553, -1.3947988748550415, 0.5118091702461243, -0.035737235099077225, -2.6864144802093506, 0.7822310328483582, -1.509635329246521, -1.1640955209732056, 0.15386039018630981, -0.0009803877910599113, 0.95475172996521, 1.1657806634902954, -0.7436821460723877, 0.1380997598171234, -0.26477938890457153, -0.9863112568855286, -0.9301965832710266, 0.3445313572883606, -0.3066818416118622, 0.902517557144165, 0.5204576253890991, -0.25213292241096497, 1.2992453575134277, -0.9170203804969788, 0.0612712986767292, 0.30481237173080444, -0.09701693803071976, -0.2228805273771286, 0.9538828134536743, 0.21091540157794952, -0.821661114692688, 0.5460135340690613, -0.7736992835998535, -0.1875159591436386, 0.7759917974472046, 2.3243966102600098, 0.2080329954624176, 1.0897314548492432, 1.5100431442260742, 0.9367160201072693, 0.04092981666326523, -0.11300548166036606, -1.6963094472885132, 0.4605688452720642, 1.852020502090454, -0.2075260877609253, -2.603377103805542, -0.5160732269287109, 1.0302629470825195, -0.782473087310791, 1.057736873626709, -0.5537919998168945, 0.12103455513715744, 2.9898746013641357, -0.3010123074054718, 1.0070226192474365, 0.04149549454450607, 0.6654526591300964, -1.919061303138733, 0.20302480459213257, -0.2669176757335663, -0.15894463658332825, 0.19024762511253357, 0.5527923703193665, -0.3629709780216217, -1.2921830415725708, -1.543715000152588, 0.8234413862228394, -1.6721065044403076, 1.6085284948349, -0.9087238907814026, -0.2596326768398285, -0.3221583664417267, 0.04488907381892204, -1.7737387418746948, 1.2901079654693604, -0.04401616007089615, -1.1853407621383667, -0.09316983819007874, 1.1583406925201416, -0.153046116232872, -0.1123957708477974, 0.4925781190395355, 0.6489049792289734, -0.310329407453537, 0.6235729455947876, 0.010242737829685211, -0.954403281211853, 1.0759530067443848, -1.3599249124526978, 1.2450813055038452, 0.7159081101417542, -1.6620230674743652, 0.9012430310249329, 0.1559334397315979, -0.12928448617458344, -1.3270376920700073, -0.3432427644729614, 1.4135030508041382, 1.3237937688827515, 0.5004750490188599, -0.5983254313468933, -0.6293563842773438, -0.854123055934906, 1.3871221542358398, -0.9649309515953064, 0.41343802213668823, 0.07909376919269562, -0.6735791563987732, 1.0376025438308716, -0.7552740573883057, -1.1325891017913818, 0.06955790519714355, -0.2355891317129135, -0.7887960076332092, -2.3038289546966553, -1.7597218751907349, -1.441809058189392, 1.6376248598098755, -2.0068204402923584, -1.9846550226211548, -0.7602728009223938, -1.466354489326477, -0.6503653526306152, 1.1383774280548096, 0.9744621515274048, -0.20738384127616882, -0.6162626147270203, 0.6614792943000793, -0.6480138897895813, 0.1739823967218399, -0.04808168113231659, -1.3531277179718018, 2.1648802757263184, -0.5678551197052002, -0.965342104434967, 1.2776036262512207, -0.2914589047431946, 0.24397985637187958, 1.1902194023132324, -0.7946048974990845, -0.5155686736106873, -1.091654658317566, 0.6797555088996887, 0.9777035713195801, -1.3084439039230347, 1.697082281112671, 0.10621877759695053, 0.004165001213550568, -1.292943000793457, 0.7577496767044067, -1.0006896257400513, 0.0052807992324233055, -0.3353743851184845, 0.9306654930114746, 1.315929889678955, 0.7425082921981812, 0.8045206665992737, -0.3613811433315277, 1.2876030206680298, -0.8752058744430542, 1.368563175201416, -0.518120288848877, 0.9019899964332581, 1.2989994287490845, -2.8776447772979736, 0.5246186256408691, -0.02301109954714775, 1.212691307067871, 0.9301444292068481, 0.6134398579597473, -0.6534020900726318, 1.4147086143493652, 1.5020763874053955, 0.6950624585151672, -2.2794594764709473, -0.44415104389190674, -0.49227389693260193, 0.6581734418869019, -0.5149256587028503, -1.6605305671691895, 0.09404546022415161, -1.1079528331756592, -1.1448051929473877, -0.9800994396209717, 0.44905245304107666, 0.4376087188720703, -0.20637890696525574, -0.5084459185600281, 1.8486521244049072, 0.15363669395446777, -0.09079521149396896, -0.32352960109710693, -0.2734001874923706, 1.774633765220642, -0.5737959146499634, -1.590155005455017, 0.19616252183914185, 0.04516894742846489, 0.274682879447937, -0.11602386832237244, 1.7788740396499634, 1.6791225671768188, -0.4626694619655609, 0.3160560727119446, -0.7583120465278625, 0.8225502967834473, -0.6460920572280884, 1.4593815803527832, 0.15057431161403656, -0.1457698494195938, -0.478521466255188, -0.19776004552841187, 0.23362010717391968, 2.231884002685547, 0.03084394335746765, 1.7704018354415894, 0.7461430430412292, -1.8346445560455322, -0.6304728388786316, -0.36433765292167664, 0.45990172028541565, -1.4812028408050537, 1.56339693069458, 1.4841456413269043, 0.5444085597991943, 1.149366021156311, -0.3911708891391754, -0.2617584466934204, -1.3139997720718384, 0.9061184525489807, 1.0599923133850098, 1.9419606924057007, -0.16804595291614532, -1.0348519086837769, -0.7258971929550171, -0.8542659282684326, -0.18062134087085724, 0.7093730568885803, 0.052985843271017075, 0.3206214904785156, -0.8343167304992676, 3.49599552154541, 0.9598597288131714, -0.8123061656951904, -0.30139675736427307, 0.21984611451625824, -1.034020185470581, -0.9277951717376709, -0.6539122462272644, -1.7209535837173462, -1.3044685125350952, -0.9678292274475098, 0.30295631289482117, 0.28242379426956177, -0.14464852213859558, 0.483183890581131, 2.2524237632751465, -0.597875714302063, -0.3300670087337494, -1.1074146032333374, 0.3805035352706909, 1.3072874546051025, 0.860339879989624, 0.5592710375785828, -0.07621189206838608, -1.0781947374343872, 1.0978550910949707, 0.5058988928794861, -0.5354905724525452, -0.8994297981262207, -0.4017183482646942, -0.03742114081978798, -0.24258413910865784, -0.5244640707969666, 0.9693669676780701, 0.7889230251312256, 1.93715238571167, -0.3548068702220917, 1.5054292678833008, 0.09052898734807968, 1.8610024452209473, -0.31721603870391846, 0.13396567106246948, 1.0922755002975464, -0.36895498633384705, -0.2782432734966278, -1.4342950582504272, -0.9342978000640869, -0.4969886541366577, -1.1004246473312378, 0.012144217267632484, 1.0191680192947388, 0.2715497612953186, 0.11928768455982208, 0.10235702991485596, -0.32306841015815735, 0.30981773138046265, 0.2760973274707794, 1.2527084350585938, -1.9800218343734741, -1.5887945890426636, -0.6256589293479919, 0.9465517997741699, 0.9724383354187012, -0.5437057018280029, -0.08321458101272583, 0.24716536700725555, 0.31950485706329346, -0.08442630618810654, -0.47701871395111084, 1.147160530090332, 1.2929524183273315, -0.5351256728172302, 0.47938525676727295, -0.38529446721076965, 0.18140093982219696, 0.292928546667099, 1.3799554109573364, 0.6188167929649353, -1.414817452430725, 1.624446988105774, 0.23573100566864014, -0.1386183351278305, 1.3069592714309692, -1.2816838026046753, -1.1709192991256714, 0.1486838459968567, 1.5254650115966797, 0.8149545788764954, 1.168175220489502, 0.682034969329834, -1.1174565553665161, 1.3454736471176147, 0.4537436366081238, -0.16510552167892456, 0.7201896905899048, -0.29289087653160095, -0.9378964900970459, 0.19208882749080658, 0.23336444795131683, -0.5473060607910156, 0.6486950516700745, -0.7120476365089417, 0.14935363829135895, -0.8118700385093689, -0.45286113023757935, -0.25573569536209106, 0.8689802885055542, -0.756723940372467, -1.233894944190979, -1.1147059202194214, -0.3220863938331604, -0.029692716896533966, 0.4933556318283081, 1.3729079961776733, 0.615811824798584, 2.5884602069854736, 0.33167192339897156, -0.6333471536636353, 0.4647771120071411, 0.8377741575241089, -1.2144845724105835, 0.9473659992218018, 0.3496764898300171, -1.0666688680648804, -0.571386456489563, 0.7892154455184937, -0.1759200543165207, -0.24375396966934204, -1.4997907876968384, 0.4277466833591461, 0.6363425254821777, -1.044019341468811, 0.5322043895721436, 2.1306381225585938, -0.22409050166606903, -1.5556764602661133, -1.1348804235458374, 0.4681951105594635, 1.8306803703308105, -0.4722630977630615, 1.2384716272354126, 0.7844609618186951, 0.12252198904752731, -1.0939263105392456, 0.4747753143310547, -0.6539187431335449, -0.4424613118171692, 1.7053258419036865, -0.5556687116622925, 0.32959598302841187, -0.9986331462860107, -0.3198230266571045, -0.9368523955345154, 1.7524080276489258, 0.5843965411186218, -0.6828300952911377, -0.14991681277751923, -2.2088277339935303, -0.9043907523155212, 1.0370744466781616, 1.510015606880188, -0.22800831496715546, 0.6553924083709717, 0.39300408959388733, -0.946982204914093, 2.2774691581726074, 0.37672796845436096, 1.5821491479873657, 1.2137662172317505, -1.6983215808868408, -1.0536788702011108, 0.7808290719985962, 0.1274915486574173, -0.08644092828035355, 0.3945472836494446, -1.115731954574585, 1.4741984605789185, -0.2755647301673889, 0.8934734463691711, 0.4504567086696625, 0.7331734299659729, -0.46491312980651855, -1.3552124500274658, -0.06407258659601212, -0.033342450857162476, -2.326324462890625, -0.4052479863166809, -1.1439675092697144, 0.3237677216529846, -1.1649800539016724, -2.056851625442505, -2.2338569164276123, -0.3392411470413208, -0.825840413570404, -0.2308514416217804, 0.5582284927368164, 0.03837265446782112, -0.6607768535614014, 1.9158470630645752, 0.13562791049480438, -0.6231397390365601, 0.869368314743042, 1.8497490882873535, 0.8331387639045715, 0.9492373466491699, 0.7950875759124756, 0.9932512640953064, -1.5327932834625244, 1.264553427696228, 0.6551787257194519, 0.3106067180633545, -0.21685969829559326, 0.15737539529800415, 0.04350690916180611, 0.9434338808059692, -0.5067331194877625, -0.03437693417072296, -0.7645314931869507, -0.3535989820957184, -0.44930651783943176, -0.5184590220451355, -0.8777726888656616, 1.6442389488220215, 0.2209448218345642, -0.17881931364536285, -0.8409640789031982, 1.762343168258667, -0.7383224368095398, 1.3436355590820312, 0.1149335578083992, 1.2233351469039917, -0.06163736432790756, 0.9571996927261353, 0.3741741478443146, -0.27748730778694153, -0.9413206577301025, 1.0895689725875854, -0.1617843210697174, 2.8745033740997314, -0.2328685224056244, -0.4499532878398895, -1.501146674156189, 0.0860731229186058, 0.8500131368637085, 0.5576891303062439, -0.08847872912883759, 1.5971038341522217, 0.7178765535354614, 1.0314046144485474, -0.5681830644607544, 1.1682034730911255, -0.8016780018806458, -0.2057129144668579, -0.7082028388977051, -1.47073495388031, -0.01865021325647831, -1.2892794609069824, -0.01624637469649315, -0.33925166726112366, -0.14246103167533875, 0.8961683511734009, -0.256422758102417, 0.3281322419643402, -0.7329482436180115, -1.4891676902770996, 0.24638570845127106, -0.8563573956489563, -1.314008116722107, 1.1377314329147339, -1.1719768047332764, 0.9596157670021057, -0.479079931974411, -0.5253584980964661, 0.6530992388725281, 2.2826058864593506, -0.5459281206130981, 0.2633614242076874, 0.16124668717384338, -1.8191237449645996, -1.3360790014266968, -0.06254854053258896, 0.765307605266571, -0.8775826692581177, -0.8993967771530151, 0.014958600513637066, 0.5156521201133728, 1.2762043476104736, 2.0983996391296387, 0.8839126825332642, -0.543554961681366, -1.2563308477401733, -0.8322304487228394, 1.1364799737930298, 2.2451388835906982, 0.884801983833313, -0.20419089496135712, -0.7571628093719482, 0.20042549073696136, -0.6871236562728882, 2.7756030559539795, 0.5102967619895935, -0.694749653339386, 0.8890858292579651, -0.685847818851471, 1.7398593425750732, -0.9568694233894348, 1.653639554977417, 0.4302879869937897, 1.4609277248382568, 0.43423524498939514, -0.5824023485183716, 0.303316205739975, 1.3409945964813232, -0.3036891520023346, 1.5655109882354736, -0.35003921389579773, 0.24768441915512085, -0.7932349443435669, 0.7439051866531372, 0.2956732213497162, 1.4870527982711792, 1.0938374996185303, -2.029263496398926, -1.124837875366211, 1.640581727027893, -0.46417877078056335, -0.35622602701187134, 0.008134731091558933, 2.177560567855835, 0.39764922857284546, 0.5786850452423096, -0.3475686311721802, 1.5559771060943604, -0.8517268896102905, -0.4575863480567932, 0.679319441318512, -1.735232949256897, 1.9399834871292114, 1.3971980810165405, 1.4624860286712646, -0.4637204110622406, 1.0235224962234497, -1.1847472190856934, 0.37971460819244385, -0.7729842662811279, -0.6799018979072571, -0.28624552488327026, 0.5532402992248535, 0.8556912541389465, 0.6284271478652954, 2.145946979522705, -0.5519497394561768, 0.726491391658783, -0.1622242033481598, 0.27970099449157715, -0.11849966645240784, 0.10656428337097168, 0.6749219298362732, -1.3168683052062988, -0.5570306777954102, 0.7125208377838135, 0.3642232120037079, -1.0683051347732544, -1.6663020849227905, 1.6408166885375977, -0.7551895976066589, 0.04918496310710907, 1.4136449098587036, 0.6718852519989014, 2.2513513565063477, -0.2404402643442154, -1.5741920471191406, 0.7414445281028748, -0.34301549196243286, -0.29427438974380493, -2.597160816192627, 0.6896764039993286, 1.9263076782226562, 1.4223301410675049, -0.2838313579559326, -1.526538610458374, -0.1577104777097702, -0.18183371424674988, -1.7654492855072021, -1.1269845962524414, -1.4205092191696167, 0.48184287548065186, 0.38378310203552246, -0.16297684609889984, 1.8577511310577393, -0.40014830231666565, 0.7508183121681213, 0.7574933767318726, -0.582276463508606, 0.23226630687713623, 1.2691380977630615, -0.7275449633598328, -0.145979642868042, -0.29146426916122437, 1.418442726135254, 0.3729276955127716, 1.2314600944519043, -0.9083620309829712, 0.5947109460830688, 0.41553622484207153, 0.630684494972229, -0.8848559260368347, -0.6489490270614624, 1.6586028337478638, 0.18221119046211243, 0.5151576995849609, 0.47248244285583496, -0.35480234026908875, 0.8512360453605652, -0.44756343960762024, -0.10003317892551422, 0.6325193047523499, 0.5458734035491943, 0.35965606570243835, -0.2855045199394226, 2.2777581214904785, 0.7865468859672546, -0.25637367367744446, -2.404825210571289, -0.9477742314338684, -0.5987429618835449, -0.18058523535728455, -0.6169716119766235, -0.1749032437801361, -0.6331585049629211, 0.3947244882583618, 2.0208349227905273, 0.7838829755783081, -1.2406539916992188, 0.6940610408782959, 0.08155238628387451, -1.4723472595214844, -0.8736549615859985, 0.27440646290779114, 1.4846196174621582, 0.17409203946590424, 2.000868797302246, 0.5372765064239502, 0.1798246204853058, 0.04449453204870224, -0.412061870098114, 1.228336215019226, -0.038603730499744415, -1.946212649345398, -0.9846173524856567, 0.049601439386606216, -1.261686086654663, -1.3199576139450073, 0.9247516393661499, 0.37654396891593933, -0.4567930996417999, -0.664659321308136, -2.881967306137085, -0.3319467604160309, -0.18617041409015656, -0.1271306276321411, 1.1410338878631592, 0.14952392876148224, 1.2626186609268188, -0.1387542486190796, 0.9248210787773132, 0.9814288020133972, 0.3010394275188446, 0.8286605477333069, 0.29810506105422974, 1.194128155708313, -0.3926096260547638, 0.06287344545125961, -0.7567620277404785, 0.17612382769584656, -0.7055420875549316, 2.2178406715393066, 2.5069994926452637, 1.1261571645736694, -1.6312379837036133, 0.5805021524429321, 0.24558155238628387, 0.9338684678077698, 0.46576598286628723, -0.10631626099348068, -0.7723641991615295, 0.10651388019323349, -0.09684935957193375, 0.6675049066543579, -1.438981533050537, -1.3572739362716675, -0.8621793389320374, -1.7200666666030884, 1.1568548679351807, -0.25123855471611023, 0.8071354627609253, -0.40019455552101135, 2.5093400478363037, 0.9349009990692139, -2.253848075866699, -0.03607434034347534, -0.08323828876018524, -0.1076626256108284, 0.5803532600402832, -0.22067086398601532, 1.4732797145843506, 1.4557727575302124, -0.14333675801753998, -0.5043028593063354, 0.012998754158616066, -0.4789879024028778, -0.21128395199775696, 0.7242268323898315, -1.9940533638000488, -0.4720902144908905, 1.1325430870056152, 1.4261858463287354, 1.3827351331710815, 0.5043259859085083, -0.17605039477348328, -0.22061152756214142, 2.4221742153167725, 0.08188769966363907, 1.6649194955825806, -1.7526946067810059, 0.4740053415298462, 0.8942570686340332, -0.19427305459976196, -2.1965627670288086, 0.674325168132782, 0.40686145424842834, -0.6546512842178345, -0.9426783323287964, 1.0994844436645508, 0.2806960940361023, 0.4417508840560913, -0.08811338990926743, -1.5347086191177368, -0.10181604325771332, 0.4130115509033203, 0.9891628623008728, -0.7444992065429688, -2.2169017791748047, 0.760164737701416, -0.07066836208105087, -0.39768335223197937, -0.8902917504310608, -0.2980458736419678, 1.12614107131958, -1.2240198850631714, 1.565322756767273, -0.12950089573860168, -1.4780495166778564, -1.1434812545776367, -0.07895132899284363, 0.4357093870639801, 0.9642569422721863, 1.9692168235778809, -0.7244822382926941, 1.6018494367599487, 0.088749460875988, -0.5139528512954712, -0.5484657287597656, -0.2469998002052307, 1.2173017263412476, 0.2772085964679718, 0.29909732937812805, -0.7633715271949768, 0.3300314247608185, -0.5029650926589966, -0.25978580117225647, -0.6528591513633728, -0.6634481549263, -0.5770975351333618, -0.3484203815460205, 0.8864812254905701, -0.4208633303642273, -0.04886850342154503, 1.1305843591690063, -0.43548595905303955, 0.31728267669677734, -1.0439056158065796, -0.07288456708192825, 1.5852391719818115, -1.4870975017547607, -0.6542389392852783, 0.9013146162033081, -0.33417588472366333, -0.10957001894712448, -0.6775542497634888, 1.2671979665756226, 0.6327730417251587, -0.6367892026901245, -0.5199105739593506, 0.8270651698112488, 1.915932297706604, 1.3019336462020874, -0.5725293159484863, 0.8310332894325256, -0.14264070987701416, 2.3563499450683594, 0.6424878835678101, -0.8965635895729065, 0.04701097309589386, 2.6705026626586914, 1.0304244756698608, 1.2298333644866943, -0.4836927354335785, 0.0761076882481575, -2.27396559715271, -2.395479202270508, 1.904017686843872, 0.45403265953063965, 2.9013888835906982, 1.833500623703003, 0.031522687524557114, 0.3292580246925354, 0.35738009214401245, -0.31371009349823, 0.28800660371780396, -0.2631029784679413, -0.8652175664901733, -0.83819979429245, 1.5216259956359863, 0.7037091255187988, -0.5946169495582581, 0.1767459213733673, 1.0685678720474243, -1.3754682540893555, 1.3408374786376953, -0.48728078603744507, 0.4744561016559601, 0.21291251480579376, -0.72235506772995, -0.6852459907531738, -0.3036406934261322, -0.9344350695610046, -0.2508837878704071, 2.4131782054901123, -0.3879149854183197, 1.0829914808273315, -0.2761979401111603, 0.4321582019329071, 1.112229347229004, -0.8970084190368652, 0.9213564395904541, 2.6080243587493896, -0.6222380995750427, -1.377822756767273, 1.0580952167510986, 1.0979760885238647, 0.06099491938948631, -1.269906997680664, -2.1446011066436768, -0.31308111548423767, 0.3038087785243988, 0.2750754952430725, 0.7887097001075745, -1.8819414377212524, -0.7528604865074158, -0.6482954621315002, -0.139863982796669, -0.9166657328605652, -0.2562100291252136, 1.3355906009674072, -0.5491304993629456, 0.23603002727031708, 0.5558537244796753, 1.302350640296936, 0.13078582286834717, -1.3835500478744507, -0.6077703833580017, 1.9792133569717407, -0.19476579129695892, 0.6156436204910278, -0.0200372114777565, -0.9855266213417053, -0.1971047967672348, -0.5673349499702454, -0.478801429271698, -0.1185123547911644, -0.5463337898254395, -0.09151167422533035, -1.972206950187683, 1.204569935798645, 0.3551551401615143, 0.11556223034858704, -0.3622385859489441, -0.5394690632820129, -0.20164261758327484, -1.5791903734207153, 0.5761200785636902, -0.20757389068603516, -0.6366848945617676, 0.533565104007721, 0.7005131840705872, 0.777759850025177, 1.0311635732650757, 1.0182011127471924, -1.393869400024414, 2.0032174587249756, 2.7024965286254883, -1.4285321235656738, -1.0575180053710938, -1.2992535829544067, -0.8631066679954529, 1.8268979787826538, 0.3477865755558014, 0.9451063275337219, 1.926995038986206, 0.9487216472625732, 0.6680575013160706, -1.3039597272872925, 0.19878438115119934, 0.6362870335578918, -1.035719394683838, 0.9214547872543335, 0.878476619720459, -1.3608390092849731, -0.5957265496253967, 0.26930177211761475, 0.9421722292900085, -0.5893615484237671, -0.07258293777704239, -0.23239533603191376, -0.5921993255615234, 0.6461563110351562, 0.4210127294063568, 0.946954607963562, -0.3187357783317566, -1.681237816810608, -0.39150816202163696, 0.7280006408691406, 0.46077877283096313, -1.9282774925231934, -0.8128890991210938, 0.18112494051456451, 1.3557754755020142, 0.4076130986213684, 0.892478883266449, 0.8683866262435913, 1.256946325302124, 0.016881905496120453, -1.414853572845459, 0.4985438287258148, 0.1951102912425995, 0.1845751851797104, -1.973425269126892, 0.2099985033273697, 1.1194477081298828, -1.0737494230270386, -0.36884695291519165, -0.7337083220481873, -0.6008032560348511, -2.3214640617370605, 0.909765362739563, 0.147731214761734, -0.9148979783058167, 1.5242058038711548, 0.8679769039154053, -2.153080463409424, -0.08765899389982224, -1.387083888053894, 0.8722970485687256, -0.6063970327377319, -1.5375609397888184, -1.324241280555725, 0.3886253237724304, -0.5527119040489197, -0.645202100276947, -0.01015970017760992, -0.35803917050361633, -0.47436246275901794, 1.0850964784622192, 1.5145376920700073, 0.3370516300201416, 1.0277252197265625, -1.2772728204727173, -0.5786278247833252, 0.24973255395889282, -1.0193122625350952, 0.2271742820739746], [-1.491089940071106, -0.036232080310583115, 0.9770268797874451, -0.12686115503311157, -1.8541152477264404, -0.7040700912475586, -0.027005432173609734, -0.3113706111907959, -0.9882904291152954, -0.3631821572780609, -0.33077356219291687, -0.4526888430118561, -2.513993978500366, -0.21325570344924927, 0.021136432886123657, 0.11854205280542374, 0.3848094046115875, 0.13377662003040314, 0.1488758623600006, 0.22868148982524872, -0.4136296212673187, -0.9680050611495972, -0.11710954457521439, -0.16747672855854034, 0.19008085131645203, -0.5139680504798889, 0.5186320543289185, 3.1122617721557617, 0.34446650743484497, 1.663243055343628, 0.9780682921409607, -0.6569182872772217, 0.5196146368980408, -0.3190585672855377, -1.5309536457061768, 0.11008169502019882, 0.9520137906074524, 0.7630472183227539, 0.3041987717151642, -1.7670701742172241, 0.027550870552659035, 0.6449074149131775, -0.21936218440532684, 0.732973575592041, 1.5998669862747192, -1.0696027278900146, 1.4616750478744507, -0.8514672517776489, 1.1555805206298828, -0.15652738511562347, 0.35656261444091797, -0.0029178932309150696, -1.1984270811080933, -0.8011773824691772, 1.4851433038711548, 0.8675395846366882, 1.0725902318954468, 0.5240910649299622, 1.3406617641448975, 1.3407255411148071, -0.23438803851604462, -1.248819351196289, -1.0342146158218384, 1.7378321886062622, -1.0440446138381958, 0.031347207725048065, 0.012917019426822662, 0.7117111086845398, 0.592277467250824, 0.6001663208007812, 0.5890348553657532, 0.6806785464286804, -0.6731488704681396, -0.8702505230903625, -0.13476087152957916, -0.14918166399002075, 0.701388418674469, -0.6767662167549133, 0.5102348327636719, 0.7437986135482788, -0.8482345342636108, 0.5928955674171448, -0.6205644011497498, 0.42630884051322937, -0.4759330451488495, -0.2764967679977417, 0.17583036422729492, 0.20516778528690338, -1.9651073217391968, 0.33830249309539795, 0.2520919740200043, 0.05592940375208855, -0.12427715212106705, -0.8970483541488647, 1.9057835340499878, -0.7331978678703308, -1.598539113998413, 1.6170363426208496, 0.754265546798706, 0.3452484607696533, -0.25056853890419006, -0.7849761247634888, -2.015261650085449, -0.9314963221549988, 0.2603869140148163, -1.5857816934585571, -0.6141430735588074, 0.13546572625637054, -0.3944270610809326, -0.29446253180503845, -0.09838961064815521, -0.6191089749336243, -0.4758153557777405, -0.2302648425102234, -0.0732542872428894, 0.5226601362228394, -2.0399067401885986, 0.8121110200881958, -1.2258833646774292, 0.924510657787323, 0.2051907330751419, 0.3688489496707916, 1.3601758480072021, 0.44968995451927185, -0.9035769701004028, -1.3789842128753662, -0.7380114793777466, -0.3545289635658264, 1.5179214477539062, -0.859138548374176, 0.48743435740470886, -0.17155949771404266, -1.2825661897659302, -2.4880831241607666, 0.3001393675804138, -1.6494619846343994, -0.6425355672836304, -0.5707064867019653, 0.3404211699962616, -0.5565862059593201, 0.5053918361663818, -0.14021487534046173, -0.6401127576828003, -2.5203113555908203, 0.7444061636924744, -0.2565445005893707, 0.9722328782081604, -0.040986254811286926, 2.2824811935424805, -0.6735424399375916, -1.3756957054138184, -0.7331483960151672, 0.1983136236667633, -0.23159730434417725, -0.9259822368621826, 0.7144845128059387, -1.7117199897766113, 1.5522013902664185, -0.37014275789260864, 0.39620593190193176, 0.18297728896141052, -1.0839800834655762, 1.6062321662902832, 0.24818523228168488, 0.08875635266304016, -0.11137503385543823, 1.652161717414856, -2.1905276775360107, 0.15429726243019104, 0.245317742228508, 1.4664262533187866, 1.0532876253128052, 1.099458932876587, -2.336512804031372, 0.24312454462051392, 0.020795919001102448, 0.7401167750358582, 0.1971559375524521, -2.3023018836975098, -0.6498486995697021, 0.1480916291475296, 0.13457345962524414, 0.6912976503372192, 0.2995128929615021, -0.6764527559280396, -0.2580154538154602, -1.2566357851028442, -0.7786850333213806, -0.34151220321655273, -0.006114330142736435, -0.4381725490093231, 0.002788886893540621, 0.8440065979957581, 0.2873072028160095, -0.9363760948181152, 0.34308892488479614, 0.39660120010375977, 0.5667033791542053, 1.4702340364456177, -1.0910650491714478, 1.073822259902954, -0.7879316806793213, 0.45822933316230774, 0.33668115735054016, -1.9604343175888062, -0.24255719780921936, 0.08529241383075714, -1.659023642539978, -0.49223530292510986, -0.24005818367004395, 0.7070788741111755, 1.821791648864746, -0.6363970041275024, 0.01784147508442402, 0.38053473830223083, 1.2480003833770752, 1.0409547090530396, -0.28579047322273254, 1.8489664793014526, 0.08135661482810974, 0.3054827153682709, -0.04232071712613106, 0.04710837081074715, -0.7447576522827148, -0.7045690417289734, 0.33531779050827026, -0.9596359729766846, 2.320312976837158, -0.9508916735649109, -0.3353925943374634, 0.8818132877349854, -0.5284637212753296, 1.0534188747406006, -0.2681291401386261, -1.0097063779830933, -0.5596963763237, -0.4919734299182892, 0.5430479049682617, 0.43601855635643005, -0.7137194871902466, -1.2847803831100464, -1.040998935699463, 2.489201068878174, -0.3160130977630615, -0.8998767733573914, -1.4083247184753418, -0.7590412497520447, -0.7172664999961853, 0.475921630859375, 0.5280901193618774, -0.3673116862773895, -1.843993067741394, -1.3739283084869385, -0.24956932663917542, 0.3865978717803955, -0.5762841105461121, 0.10953477025032043, 0.2566761076450348, -0.15762457251548767, 0.4962431788444519, 0.7131398916244507, -0.31909212470054626, -0.6963064074516296, 0.2417663186788559, -1.3192039728164673, -1.1105397939682007, -0.8007170557975769, -0.340242475271225, 1.3820959329605103, -0.025836313143372536, -1.3758803606033325, -0.9944155216217041, 0.07734441757202148, 0.6476055979728699, 0.4500964283943176, -0.3679277300834656, -1.0833157300949097, 0.15761421620845795, -1.8205386400222778, 1.6422500610351562, -0.6331784725189209, -2.8474340438842773, -0.9298370480537415, -0.5584113001823425, 1.1925307512283325, -1.4267195463180542, -0.5551453828811646, 1.6492040157318115, 2.209872245788574, -0.7810349464416504, -0.07409676164388657, -1.4149513244628906, 1.307068109512329, -1.030024528503418, 1.0075594186782837, -0.3647692799568176, 1.8704757690429688, -0.06557594984769821, -0.7060434818267822, 1.4544917345046997, 1.14786696434021, 1.0630494356155396, 0.16433946788311005, -1.5605703592300415, 1.3142510652542114, -0.918999195098877, 1.1324101686477661, 0.5366497039794922, 0.6325253248214722, 0.8954112529754639, -0.5029663443565369, 0.8036335110664368, 0.6315340995788574, -1.6064014434814453, -0.3666519820690155, 2.04292631149292, 0.1436038613319397, 0.7089048027992249, 1.0807476043701172, 1.021374225616455, -1.2418183088302612, 0.3566576838493347, 0.995453953742981, -3.1075143814086914, 0.612591028213501, 1.3668713569641113, -1.0560821294784546, 0.9272834062576294, 0.07416315376758575, -1.2260963916778564, 0.44582727551460266, -1.4378753900527954, -2.9669902324676514, -0.6563174724578857, 0.6396563649177551, -0.42131727933883667, 0.8355168700218201, -1.4974666833877563, 1.2227011919021606, 0.981225848197937, -0.7256045341491699, 0.1766846626996994, -0.037557680159807205, 0.6970418691635132, -0.9507019519805908, 0.469866007566452, 2.368528127670288, -1.2738791704177856, -0.7860713601112366, 0.9578446745872498, 1.0958614349365234, -0.5151950120925903, 0.6614497900009155, -0.49200963973999023, -0.045390963554382324, 1.9823395013809204, 0.5615862607955933, -2.129338264465332, 0.25246524810791016, -1.2821940183639526, -0.28748181462287903, -0.2855808138847351, -1.1480518579483032, -0.24301205575466156, -0.10916047543287277, 2.0287833213806152, -2.0302951335906982, -0.6031970381736755, -0.26957809925079346, 1.4317697286605835, 0.38752323389053345, 0.13827580213546753, 0.22559227049350739, -2.113677978515625, -2.0551254749298096, -1.5586564540863037, 0.25328385829925537, 1.2472708225250244, -0.1916971504688263, 0.47899919748306274, -0.6539826989173889, -0.9356330037117004, -1.1527765989303589, -2.5108299255371094, 1.636024832725525, 1.1590219736099243, -0.9915519952774048, 0.0720219612121582, -0.025264643132686615, 0.06529746949672699, 1.6254621744155884, -1.8731513023376465, 0.3849201500415802, -0.012462137266993523, -0.22744707763195038, -1.5201809406280518, -0.11003889888525009, 1.9562371969223022, 0.644625186920166, -2.5545623302459717, 2.4142918586730957, -1.2005913257598877, -2.1290557384490967, 0.8924970626831055, 0.9910039305686951, 1.1336474418640137, 0.810944139957428, -0.8011921048164368, 0.5692696571350098, -0.22452059388160706, -0.6527175307273865, 0.880246102809906, 1.851019024848938, 2.12854266166687, 1.5153367519378662, -0.6671914458274841, 1.6828339099884033, 0.7921833992004395, 3.225984573364258, -0.24126482009887695, 0.7739985585212708, -0.34210771322250366, -0.027151288464665413, 0.33803999423980713, 0.7348294258117676, 0.2757815420627594, 1.4788416624069214, -0.1386399120092392, -1.6020492315292358, 1.000644564628601, -0.6084476113319397, -0.8640273213386536, -0.31549906730651855, -0.3651105463504791, 0.2960413098335266, -0.22408142685890198, 0.225457102060318, -0.8717867732048035, 0.8383380174636841, 0.5946231484413147, -0.018659833818674088, -1.2656983137130737, 1.2833101749420166, -1.6425920724868774, 0.3107451796531677, 0.9808380007743835, -1.602839708328247, -0.10090329498052597, 1.4424090385437012, -1.215850830078125, -0.32674381136894226, 0.47020331025123596, -0.5346575975418091, -0.8518141508102417, -0.3234207034111023, 0.3019641935825348, -1.3243039846420288, 1.6742390394210815, 1.5982706546783447, 1.179036259651184, 1.3980058431625366, -1.5737591981887817, 1.5897622108459473, -1.0194860696792603, 0.4553559720516205, 0.006074924021959305, -1.343502402305603, -1.7178436517715454, 1.3363929986953735, -1.1343618631362915, -0.8861680030822754, -0.25229355692863464, 0.8653441667556763, 0.9907619953155518, 1.0497170686721802, 0.06475866585969925, 0.6485651731491089, -1.5234664678573608, 0.27600663900375366, 1.0445901155471802, 0.08494342863559723, -0.6615797281265259, -0.8288795351982117, 0.8283421993255615, 0.2425316721200943, -2.0401182174682617, -0.43805932998657227, -0.45773810148239136, -0.6288608908653259, 0.6243410706520081, -0.6485369205474854, -0.40018054842948914, 0.9353659749031067, -0.5856838226318359, -0.14226113259792328, -0.3444717526435852, 1.85537588596344, 0.5918830633163452, 1.034960389137268, -0.24024349451065063, -0.7274675965309143, -1.4001052379608154, 0.08625106513500214, -1.7160199880599976, 0.4433805048465729, -0.06389868259429932, -0.3737946152687073, -0.22271181643009186, 1.1352860927581787, 0.10230594128370285, 0.3730504512786865, -1.632150411605835, -1.8796043395996094, 0.5083340406417847, -1.6646268367767334, 0.613379716873169, 0.4824274480342865, -1.2768356800079346, -1.9049373865127563, 1.3048653602600098, 0.40911394357681274, -1.3967653512954712, -0.22464019060134888, 0.03659558296203613, 0.8348619937896729, -0.7254431843757629, -0.3985017240047455, 1.06105637550354, 1.0052704811096191, 0.6116280555725098, 1.5468511581420898, 2.176454544067383, 1.3064924478530884, 0.2623537480831146, 0.17992334067821503, -1.5750291347503662, 0.29492998123168945, -1.0008580684661865, -1.193198323249817, -0.2857625484466553, -0.6274336576461792, -0.5891473293304443, -1.4749921560287476, -0.8129318356513977, 0.4631824195384979, -0.8062537312507629, 0.4281989634037018, -0.07348689436912537, 1.493425726890564, -0.6407947540283203, 0.25639212131500244, -1.282312273979187, -0.017213262617588043, 1.956804871559143, -1.0934287309646606, 0.5790296196937561, -1.5703272819519043, -0.020113730803132057, -0.12889041006565094, 0.5520066618919373, 0.20688775181770325, 0.17282916605472565, -0.23245790600776672, 1.6934176683425903, -0.8132246732711792, -0.04605557397007942, -0.6359350681304932, 1.1558247804641724, -0.15161874890327454, -1.1844173669815063, 1.0756579637527466, -2.318258047103882, 1.137846827507019, -1.4668190479278564, 0.5174520611763, -1.594954252243042, -0.5031713247299194, -2.6509408950805664, 0.3244612216949463, 0.5999093651771545, -2.205815553665161, 0.23302480578422546, -0.3985145092010498, -0.9477508664131165, -0.8639854192733765, -0.18990673124790192, 1.2838242053985596, 0.8648863434791565, 0.11772961914539337, 0.2706378102302551, 0.9621760845184326, 0.4659237861633301, -1.6519689559936523, 0.06031493470072746, -0.6402540802955627, -1.6815547943115234, -0.3829438090324402, 0.42483559250831604, 0.26118966937065125, 2.024290084838867, -0.8634181618690491, -1.4720851182937622, 0.26646170020103455, -0.7440940141677856, -0.6706584692001343, -0.2711602747440338, -0.7929747104644775, 0.9154155254364014, 0.487334668636322, 0.7388858199119568, 1.564627766609192, -0.5285771489143372, -0.332093209028244, -0.33304300904273987, -0.09131771326065063, -0.8131735920906067, -0.5375032424926758, -0.11884935945272446, 1.9841132164001465, -0.2843608558177948, -0.5070884823799133, -0.09159877151250839, 0.7485715746879578, 1.295678734779358, -1.8870164155960083, -0.3933745324611664, 1.0961672067642212, -0.6752669811248779, 1.9229704141616821, 0.9337966442108154, -0.005901162978261709, -1.9123185873031616, 1.0195719003677368, -0.6352614760398865, -1.0790947675704956, 1.5901747941970825, 0.08152135461568832, 1.1202776432037354, 0.3644978404045105, 0.3349526822566986, 1.5213004350662231, -0.2299726903438568, -0.3512800335884094, -2.204777240753174, 0.5596514344215393, 1.3562707901000977, -0.5477015376091003, -0.3947657644748688, 0.9816399216651917, 0.02128734439611435, -0.5845577120780945, 0.7373583316802979, -0.03679294511675835, -1.6707199811935425, 0.7191522717475891, 0.2937869727611542, 0.4618760347366333, -0.12780481576919556, -0.8892902731895447, -0.4966399073600769, -0.48892417550086975, -0.45885977149009705, 0.1758182793855667, 0.626063346862793, 0.13284417986869812, -1.8287365436553955, 0.6790648698806763, 2.02217960357666, -0.7830651998519897, 0.38954415917396545, -0.6420596241950989, 0.9352856278419495, -1.355233907699585, 0.10692276805639267, 0.4813529849052429, -1.8797800540924072, -2.395690679550171, -1.7913538217544556, -0.7264344692230225, 1.7639238834381104, 2.149839401245117, 0.7290260195732117, 1.2340466976165771, -1.7426567077636719, 0.5996447801589966, -0.09432895481586456, -1.7777435779571533, -0.21494129300117493, -0.8383645415306091, 1.3378632068634033, -2.242234706878662, 0.5277330875396729, -0.9007480144500732, 0.1155586764216423, -0.5524914264678955, 0.23811522126197815, 0.3514561057090759, -0.6057887673377991, 2.186941146850586, -0.25449931621551514, -1.1350915431976318, -1.0451714992523193, 0.14279404282569885, -1.119081735610962, 0.49117010831832886, -0.6038562059402466, -1.4426143169403076, -0.26384320855140686, -0.19109107553958893, -0.6679835319519043, 0.7692883014678955, -1.6100597381591797, 0.37007972598075867, 0.4352571368217468, -2.0063114166259766, -1.4626609086990356, -0.6846410036087036, -0.596547544002533, -0.6345075964927673, -0.5780783891677856, -1.6539181470870972, -1.14641273021698, 0.824088454246521, -1.9925172328948975, -0.9979575872421265, -0.7867969274520874, 1.4847438335418701, -0.4268605411052704, 0.2934030592441559, -1.001140832901001, 1.2331737279891968, 0.28970813751220703, 2.3448312282562256, -0.5293316841125488, 0.04729176312685013, -0.6100148558616638, -0.46233904361724854, 1.3220549821853638, 1.2482742071151733, 0.8952779769897461, -0.461660236120224, 0.8504889607429504, -0.852593183517456, 1.0658719539642334, -0.033042218536138535, -0.35972800850868225, -0.811074435710907, 0.6206839084625244, -1.5458064079284668, 0.6273590922355652, 0.18264459073543549, -1.3740379810333252, 1.6580936908721924, -1.1102851629257202, 0.18576376140117645, -0.10071073472499847, 0.7763161063194275, -0.3301961421966553, -1.0378729104995728, 0.7293650507926941, 0.7810776233673096, -0.0708426833152771, -0.880831241607666, 1.014263391494751, 0.5219138860702515, 2.2792556285858154, 2.4259021282196045, 0.2769206166267395, -0.09335076808929443, 0.7560080885887146, 2.145139217376709, -0.145895853638649, 1.9037120342254639, -0.13578611612319946, 1.175345778465271, 0.2226630002260208, 0.3590368330478668, 1.9349361658096313, -0.7970484495162964, 0.830407440662384, -0.09902238845825195, -1.6328431367874146, 0.9201768040657043, 1.279049277305603, 0.15146996080875397, 2.175698757171631, 1.2385146617889404, -0.3392621874809265, 0.9633671641349792, -0.6476454138755798, 0.09954995661973953, -0.7595570087432861, -0.15312640368938446, -0.003976247739046812, -0.718084454536438, -0.35606256127357483, -2.008406400680542, 0.7445145845413208, -0.09659789502620697, 0.0971759781241417, 0.8942123651504517, 0.44324326515197754, -0.1842534989118576, -0.8918197751045227, -0.7290727496147156, 0.5930632948875427, -1.5851802825927734, -1.5282392501831055, 0.2361963987350464, -0.4197290539741516, 0.3397238850593567, 0.3737814724445343, 0.47858721017837524, 1.006243348121643, -0.9335646033287048, 0.17007946968078613, 1.931888222694397, -0.5748202800750732, -0.8099275231361389, 0.8258859515190125, 1.1756014823913574, -1.28135347366333, 0.9696758389472961, 0.6916697025299072, 0.593413233757019, 1.2674405574798584, -0.025200961157679558, -1.9571714401245117, 0.5618590712547302, -0.17019544541835785, 1.592149019241333, 0.009349508211016655, 0.16809441149234772, 0.435689777135849, 1.6643646955490112, -0.9475404024124146, 0.18666906654834747, -0.34856078028678894, -0.38118767738342285, -0.5662528276443481, 0.2814767360687256, -0.46987369656562805, 0.2911732792854309, 1.2114850282669067, -0.5863389372825623, 0.7512823343276978, -0.5319873094558716, -0.006787389982491732, 0.12305013835430145, 0.9225811958312988, 1.1461995840072632, -1.685377836227417, 0.637403130531311, -0.45265090465545654, 0.7316807508468628, 1.149376630783081, 0.5412473678588867, -1.5556180477142334, -0.1914362609386444, 0.7024134397506714, 0.10629138350486755, -1.4052802324295044, -1.2679579257965088, 0.43558165431022644, -0.008560506626963615, -1.2202731370925903, -0.2594377398490906, -0.5694133043289185, -0.8541014194488525, 0.8056321144104004, -0.10161522030830383, 0.21847154200077057, 0.3758850395679474, -0.10176202654838562, -0.7696437239646912, -1.163915753364563, -0.729398250579834, 1.7787420749664307, -0.8010275363922119, 0.2122337520122528, 0.5248852968215942, -0.5509886145591736, -0.3230532705783844, -0.3868578374385834, 1.8736209869384766, -0.9348161220550537, 0.917056679725647, -0.04730295017361641, 1.2557761669158936, -0.6991963982582092, -1.200311303138733, -0.914531409740448, 0.04204505681991577, 0.7219347953796387, -0.9675971865653992, -1.5608233213424683, 0.06619076430797577, -1.6317012310028076, -1.2607572078704834, -0.11953867226839066, 1.3704547882080078, -0.4159296751022339, -0.3670348525047302, 2.312324047088623, 0.4768969714641571, 0.7922130823135376, -0.9408102035522461, -0.12379862368106842, -0.7153109312057495, 0.6827591061592102, -0.1702021062374115, 0.9785045981407166, 1.7002209424972534, -0.9894145727157593, 0.6585747003555298, 0.7489452362060547, -0.6678857803344727, -3.4600303173065186, -0.4582533538341522, 0.4102559983730316, -1.606344223022461, 1.548668622970581, -0.9947412610054016, -0.09186959266662598, -0.6450071930885315, -0.5195189118385315, 1.4716869592666626, -0.15282662212848663, 0.8904740214347839, 0.6110119223594666, -0.1160321831703186, 0.15111872553825378, 1.5515550374984741, 0.29734158515930176, 0.6609986424446106, -0.33784735202789307, -0.5739068984985352, -1.3686612844467163, 1.3337714672088623, -1.2795794010162354, -0.2948373854160309, -0.20755983889102936, -2.3962230682373047, -0.11738106608390808, 2.29943585395813, 0.7724036574363708, 1.1868505477905273, 1.5335817337036133, -0.3013176918029785, -1.0434629917144775, 0.702613353729248, 1.5911222696304321, -0.6703857779502869, 0.6855501532554626, 0.7473645806312561, 0.9400970935821533, 1.7596967220306396, -1.680358648300171, 0.3721866309642792, 1.2484126091003418, -0.33660414814949036, 0.16511303186416626, -0.46996578574180603, -1.7776458263397217, 0.36119356751441956, -0.8101596236228943, 0.20677052438259125, -0.8017281293869019, -0.8695690631866455, 1.0911628007888794, -1.1027888059616089, -0.5564031004905701, -0.11342452466487885, -0.4914998710155487, -1.2272032499313354, 1.153757095336914, -0.21494579315185547, 0.7017106413841248, 0.5925625562667847, 0.4291849732398987, -0.6045061349868774, 0.37051230669021606, 0.0837295651435852, -0.3858788013458252, 0.11124978959560394, 0.23055262863636017, -0.2113252580165863, -0.030997183173894882, -0.5684002041816711, -0.7567549347877502, 0.31266161799430847, 2.394087791442871, 0.781266987323761, 2.26832914352417, 1.2704086303710938, 0.35898590087890625, -0.6140292882919312, 0.18806301057338715, 1.10463285446167, 0.19180530309677124, 0.9610661864280701, -2.65815806388855, 0.4342733323574066, 0.30017802119255066, -0.3123595416545868, 0.00687401182949543, 0.5001617074012756, 3.00541615486145, -0.5003753900527954, 0.30194249749183655, 0.7048646211624146, 1.6831245422363281, 1.7408766746520996, 1.8766629695892334, -0.2774931490421295, 0.6220066547393799, 0.100746288895607, 0.32202836871147156, 0.12916864454746246, -0.7436781525611877, -1.1269874572753906, 1.8649030923843384, -0.0042781829833984375, 0.7600168585777283], [0.7897680997848511, 0.4449765980243683, -0.5869321823120117, -0.43633341789245605, -0.8926243782043457, 1.7460994720458984, -1.7784217596054077, 0.7553002238273621, -1.5259547233581543, 1.2445427179336548, -0.16501230001449585, -0.9440670013427734, -1.8895763158798218, 0.33071357011795044, 1.2431282997131348, 1.3178521394729614, 1.0244717597961426, 1.8538604974746704, -1.002963662147522, 0.3352542817592621, 0.02365517057478428, 0.41178181767463684, 0.5177629590034485, 0.40268582105636597, -0.2755148112773895, -0.8045508861541748, 1.9079586267471313, -0.47443249821662903, 0.996377170085907, -0.34507447481155396, -1.69512939453125, 1.0251920223236084, -0.5829930901527405, 0.2765394449234009, -0.8453337550163269, 0.041317470371723175, -0.9290207624435425, 0.4884338080883026, 0.9199767708778381, 0.4159834682941437, -0.9864076375961304, 0.408252477645874, -1.3753013610839844, 0.12154672294855118, 0.8280001878738403, 1.0930378437042236, 0.7600033283233643, 0.3443223237991333, 1.3754054307937622, 0.4436676800251007, 0.7830800414085388, 0.9314409494400024, 0.21673516929149628, 0.48716139793395996, -0.1195932924747467, 0.5959810018539429, 1.0394983291625977, 1.1094902753829956, -0.42707714438438416, -0.617107093334198, 1.4579145908355713, -1.568264365196228, 0.9006112813949585, 0.6826457381248474, -1.7621591091156006, 2.476146936416626, 0.37269827723503113, -1.1127170324325562, -0.9689862132072449, -2.276428699493408, -0.8878251314163208, -1.8511830568313599, -0.08833517134189606, 0.09381087869405746, -0.7684639692306519, -0.4183356761932373, -0.354162335395813, -0.07439859956502914, -0.004550722893327475, 2.550323724746704, -0.18311111629009247, 0.6731644868850708, -0.4974629878997803, 1.345350980758667, 1.0854240655899048, -1.6730940341949463, 0.8357596397399902, -0.19733086228370667, -0.3450510501861572, -1.3470911979675293, 0.5616623759269714, -0.22581341862678528, -0.4807854890823364, -0.3435361385345459, 0.3834967017173767, -1.2264882326126099, 0.03690755367279053, -1.1051595211029053, -0.6285955905914307, 0.847248911857605, -0.8382168412208557, 0.033691540360450745, 0.4045659899711609, -0.008955917321145535, -0.6849068403244019, -0.1436687558889389, -0.40139254927635193, -0.129093736410141, -0.6809096336364746, -0.17560897767543793, -0.30198267102241516, -0.3115104138851166, -0.9980039596557617, 0.5921768546104431, 0.3049602806568146, 0.11330832540988922, 0.14282208681106567, 0.16275759041309357, -0.20609912276268005, 0.47695863246917725, 1.2270818948745728, -1.5509434938430786, -0.05194878950715065, 0.9142067432403564, 0.7129508852958679, 0.6267866492271423, 0.8788390159606934, -0.004905236419290304, 0.3239501118659973, -0.6245889663696289, -0.15062859654426575, -0.046163447201251984, -1.455405354499817, -0.4399895668029785, 0.8414182662963867, 0.41770491003990173, 0.16322357952594757, -0.1892388015985489, 0.0018524914048612118, 1.375715970993042, 1.6771001815795898, 0.24750544130802155, 1.933735966682434, -2.4181275367736816, -1.9485599994659424, 1.4353622198104858, -0.8509715795516968, 0.08059827238321304, -0.40500059723854065, 0.6567950248718262, -1.1107754707336426, -0.6191667914390564, -0.3407646119594574, 0.7686691880226135, 0.9049050807952881, 0.03710265830159187, 0.544141948223114, -0.17240990698337555, 0.8833016753196716, 0.10168906301259995, -0.2190060019493103, -0.06057063490152359, 1.629196047782898, 0.14626038074493408, -0.6073555946350098, -0.8377036452293396, 0.43303731083869934, 0.38606175780296326, 0.35704848170280457, 1.9310046434402466, -0.3851718306541443, -1.6364637613296509, 0.49911701679229736, 1.0255684852600098, 0.7088565826416016, 0.9944884181022644, 0.19402413070201874, 2.0525808334350586, 1.7562828063964844, 0.6341880559921265, -1.522822618484497, 0.5137069225311279, 0.2228725254535675, -0.00260329176671803, 0.2644176185131073, -0.3949403166770935, 1.7547719478607178, 1.603581190109253, 1.8808513879776, -2.2923896312713623, -0.9904851913452148, -0.5171377658843994, -0.003726938273757696, 2.2391247749328613, 2.4618518352508545, -0.14618630707263947, 0.6391293406486511, -0.29833346605300903, 0.8438951373100281, 1.8125488758087158, 0.3426748216152191, 0.9305465817451477, 0.20791840553283691, 1.5978072881698608, 0.36411163210868835, 1.7738116979599, 1.178586483001709, -0.16565261781215668, 1.211975336074829, -1.3579411506652832, 0.6717009544372559, 0.4202231466770172, -0.4631425142288208, -0.06102769076824188, 1.7419153451919556, 0.23506563901901245, -1.2143832445144653, 0.7835756540298462, 0.3543463349342346, -1.0071953535079956, 0.5954965353012085, -1.8040987253189087, -0.5513294339179993, 0.6759182214736938, -0.5990762114524841, -0.6707807779312134, 0.23459817469120026, 0.16202674806118011, 0.08594577014446259, 1.3030545711517334, -0.26598766446113586, 0.9739030003547668, -0.533473789691925, 0.04050885885953903, 0.00760359363630414, -1.8630462884902954, 0.8617087006568909, 1.6112290620803833, 0.5315083861351013, -0.13950639963150024, 1.8338935375213623, 1.6696058511734009, -2.4634315967559814, -1.3629041910171509, 0.9937301874160767, 0.1754809468984604, 0.04407323896884918, -1.8745356798171997, -2.9408998489379883, -2.141602039337158, 0.5306018590927124, -1.8433780670166016, 1.3624221086502075, -0.31367751955986023, 2.373291492462158, 1.2239131927490234, -0.0035397810861468315, -0.5159839987754822, 0.02416219748556614, 0.6925805807113647, -1.7717838287353516, 2.245490789413452, 0.6013761162757874, 2.3902101516723633, 0.4729907512664795, -0.811747133731842, -0.5268771648406982, -0.9382361173629761, 1.963154673576355, 1.482223391532898, -2.0733063220977783, 0.06785447150468826, 0.6568092107772827, -0.49915677309036255, 1.452925443649292, 0.38023874163627625, 1.2664769887924194, -2.543491840362549, -1.360085368156433, -0.11215529590845108, 0.48841044306755066, 1.2501487731933594, 1.3364858627319336, 0.5968950390815735, 0.6311670541763306, 1.1568348407745361, 1.3628774881362915, -0.6535496115684509, 0.6816335320472717, -0.3919993042945862, 1.1876760721206665, 0.9097962379455566, -0.6886065006256104, 0.7753105163574219, -2.419321298599243, -0.6407265663146973, -0.8593280911445618, 0.5232216715812683, -0.27402088046073914, 1.3125635385513306, 0.7123324871063232, -0.5270213484764099, -0.972981333732605, 0.4723082482814789, 1.091578722000122, 0.9708343148231506, -0.7468166351318359, -1.9230949878692627, 0.16018778085708618, 1.5125880241394043, -0.15336190164089203, -0.8879439830780029, -0.33118364214897156, 1.040122628211975, 1.315177083015442, -0.0014228919753804803, -1.0559642314910889, 0.19878356158733368, 0.2466294914484024, 0.13656438887119293, 1.508880853652954, 0.8817161321640015, -0.4434126317501068, 0.6890610456466675, 2.080620288848877, 0.8295814990997314, 0.5025206804275513, -1.1666139364242554, -1.6520603895187378, 1.9059562683105469, 0.04897158220410347, -0.00116554438136518, -0.16325169801712036, 1.2673393487930298, -0.19070616364479065, -1.1415040493011475, -1.5367196798324585, 1.7605841159820557, -0.5987640023231506, -0.3994717299938202, 0.6173601150512695, 0.13670766353607178, 1.19295072555542, -0.4525722861289978, 0.09874676913022995, 0.2629280686378479, -0.8189845681190491, -1.0059744119644165, 0.17696402966976166, -0.8158310651779175, -2.0460731983184814, 0.32882383465766907, 0.5807786583900452, -0.3794328272342682, -0.8755752444267273, -0.4722425639629364, 2.2744600772857666, 0.21882569789886475, -0.5564706921577454, -0.12177467346191406, 1.6040031909942627, -0.5109981298446655, -0.9018855690956116, -0.7538753151893616, -1.7749550342559814, -0.5722606778144836, 0.4488687813282013, -0.5553164482116699, 0.2611912488937378, 0.49866896867752075, 2.2412116527557373, -0.004192480817437172, 0.7641754150390625, -1.8556197881698608, -1.0337879657745361, -0.541576623916626, -0.5034365057945251, -0.8178128600120544, 0.20132949948310852, -0.9608532786369324, 1.195586085319519, -0.7954333424568176, 0.06438476592302322, 0.9320374131202698, 0.4451608657836914, 0.7478504180908203, 0.41506239771842957, 0.04294879361987114, -0.7421451210975647, -0.44114458560943604, 0.6763579845428467, -1.050355315208435, -0.20799033343791962, -0.4699974060058594, -1.203969120979309, 0.36585330963134766, 1.7861950397491455, 0.21479453146457672, 1.5339808464050293, -0.09720563143491745, -0.1818438023328781, -1.6244614124298096, -0.9673463106155396, 1.528546929359436, -0.15873990952968597, 0.29801368713378906, 0.8561229109764099, -0.8219779133796692, -0.6848124265670776, -1.0137050151824951, 0.7462743520736694, -1.4993672370910645, 0.3716524541378021, 1.8338032960891724, 0.3004768490791321, 0.979951024055481, 0.7392586469650269, -0.6179755330085754, 0.09142247587442398, 0.7601436376571655, 1.4920244216918945, 2.4680864810943604, 0.7088704705238342, -0.47701317071914673, 0.34127187728881836, 1.15017831325531, -0.3127939701080322, -0.38138556480407715, 1.646960735321045, 0.4112863838672638, -0.247024804353714, -0.7100110054016113, 0.5205643177032471, -0.48225492238998413, 0.2893308103084564, -0.02694607526063919, 0.8622863292694092, -1.6102303266525269, 1.3478507995605469, -0.5131053924560547, -1.7168735265731812, 0.3818885385990143, 0.621640145778656, 1.0516290664672852, -0.7183206081390381, 0.7280932664871216, -0.8008299469947815, 1.3092893362045288, -0.5735130310058594, 0.1464637666940689, 0.6558969616889954, -0.4888991415500641, 0.7145265936851501, 0.8590621948242188, -0.5946270823478699, 1.029009461402893, -0.5418747663497925, -1.5435142517089844, -1.2066373825073242, 0.2912742495536804, 0.021069273352622986, 0.8061287999153137, 1.316951036453247, 1.078470230102539, -1.343395709991455, -0.9694721102714539, 0.20152375102043152, 2.9338831901550293, 1.5345590114593506, -0.05795857310295105, -0.0837087407708168, -1.8493475914001465, -0.3848852515220642, 1.3137633800506592, -2.368828535079956, 0.6450355052947998, -1.0426093339920044, -1.1704590320587158, 0.4906454086303711, -1.2032010555267334, 1.006149172782898, -1.06473708152771, 0.5168315172195435, -0.014875054359436035, 0.6700904369354248, -0.31073611974716187, -0.8717797994613647, 0.5779243111610413, 1.9176697731018066, -0.45521658658981323, -1.1676667928695679, 0.5934762954711914, 1.3028758764266968, 0.10855060815811157, -1.8063764572143555, -2.7351341247558594, -1.233922004699707, 1.8605759143829346, 0.9319614171981812, 0.6030884385108948, -0.5020866990089417, -0.43946918845176697, -1.207580804824829, -1.3924130201339722, 1.037196397781372, 0.5646275877952576, 0.13440267741680145, 0.8286512494087219, 1.2529375553131104, 0.32412800192832947, -1.2864278554916382, 1.1200844049453735, 1.4379013776779175, 0.3867796063423157, -1.0028210878372192, 0.5072785019874573, -1.3748321533203125, 0.7518019676208496, -0.841888964176178, 0.0862341895699501, -0.045972418040037155, -1.6238656044006348, -1.612353801727295, -0.6217090487480164, -0.2000538408756256, 0.42822396755218506, -0.49714502692222595, -0.6065307259559631, 0.5181607604026794, 0.24470843374729156, -0.6970480680465698, 0.3859502673149109, -0.8635945916175842, 0.6492573618888855, 0.6456454396247864, -1.5678515434265137, -1.469941258430481, 0.6007325053215027, 1.3589050769805908, 0.1731201410293579, 0.6877623796463013, -1.5267773866653442, 0.7897741198539734, 1.101262092590332, 1.9609718322753906, 0.08455653488636017, 1.1799187660217285, 0.40398791432380676, 0.9933863878250122, -0.5466039180755615, -0.10061357915401459, 1.4722005128860474, -1.830428957939148, -0.43637850880622864, 0.7184221744537354, 0.3249886929988861, -1.5133824348449707, 0.470990926027298, 0.17294660210609436, -1.2281403541564941, 0.38654565811157227, -0.32649505138397217, 1.3143260478973389, -0.39368748664855957, -0.9028158187866211, 0.3397175073623657, 0.8721045255661011, -0.8122561573982239, 1.0473781824111938, -0.6934059858322144, -0.20964132249355316, 0.4144555330276489, 0.6096525192260742, -2.0514488220214844, -0.2687082290649414, -1.0331968069076538, -0.22113825380802155, 0.6432982087135315, -1.3703019618988037, 1.4134465456008911, 1.087886929512024, -0.5560442805290222, -0.3695175051689148, -3.291766881942749, -2.1519110202789307, -0.006164513528347015, -1.0715419054031372, 0.3715558350086212, 2.444999933242798, -0.5638275146484375, -1.2239880561828613, -0.728550374507904, -0.22642935812473297, 0.5919771790504456, 1.3095394372940063, 0.8203589916229248, -1.0591708421707153, -1.33757483959198, 1.7980272769927979, -0.5868076086044312, 0.29168736934661865, -0.756878137588501, 0.7378393411636353, -2.2597038745880127, 1.2416778802871704, -0.15988430380821228, -1.725178599357605, 1.1940637826919556, 0.3073046803474426, 0.3449517786502838, -1.5094958543777466, 1.182167649269104, 1.4715558290481567, -1.0154253244400024, -0.8353256583213806, -0.16651593148708344, -0.9043728709220886, 0.4932633638381958, 0.03983134403824806, 1.8593230247497559, -3.0664222240448, -0.8913455605506897, 0.3689836859703064, -0.20953699946403503, 1.248304843902588, 0.11130660027265549, -2.1082561016082764, -0.704361617565155, -0.6012486815452576, 1.1177241802215576, 0.2008594423532486, 1.0642985105514526, 0.6919205784797668, 1.151883602142334, 1.144023060798645, 0.10496891289949417, 0.8821322917938232, 1.0560338497161865, 1.4548593759536743, 2.1314659118652344, 0.1450866460800171, 0.5654613971710205, 1.01475989818573, 0.046342652291059494, -1.263386845588684, -0.9121167063713074, -0.3897603452205658, -0.12087707966566086, 0.5576561689376831, 2.3762266635894775, 0.9819720983505249, 0.30731508135795593, 0.7593908905982971, -0.5313532948493958, 0.18018321692943573, -0.572777509689331, 0.8914940357208252, -1.447701096534729, -1.6876490116119385, 0.08540777117013931, 2.4784915447235107, -0.044219668954610825, 1.5166802406311035, 1.7832297086715698, 2.6565804481506348, -0.1968584954738617, 0.11414024233818054, 0.5230785012245178, 0.6283594965934753, -0.46553274989128113, 0.26660946011543274, 0.3082067668437958, 0.34125015139579773, -0.4424142837524414, -0.17663730680942535, -1.9916247129440308, -0.32427704334259033, 0.02910558134317398, -0.03325602412223816, 0.48194465041160583, -0.6965956687927246, 0.7245053052902222, 0.23048390448093414, 0.08921042829751968, 0.14018307626247406, 0.9526824951171875, 1.3513436317443848, 1.30223548412323, -0.43596500158309937, -1.1307141780853271, -0.27519941329956055, 1.7618671655654907, -1.0707536935806274, -0.10895684361457825, -0.6820494532585144, -1.1208516359329224, 1.4242866039276123, -0.6795494556427002, -1.3230180740356445, 0.3710685968399048, -0.08858341723680496, -1.3965442180633545, -0.849254846572876, 1.297343373298645, -0.8394569754600525, 3.651531219482422, 2.1654200553894043, -0.7461593747138977, 0.03276916220784187, -0.06350678205490112, 0.34094706177711487, -0.3463399410247803, 2.1967132091522217, -1.0099471807479858, 1.6404807567596436, 0.795958399772644, 1.2872238159179688, -0.23706504702568054, -0.6401579976081848, 0.690066397190094, 0.36395812034606934, 0.971547544002533, -0.14034174382686615, -0.9242181777954102, -0.40999358892440796, -0.39690661430358887, 0.37861236929893494, -2.4024977684020996, 0.8079078197479248, -2.7348482608795166, 0.014831982553005219, -1.248258352279663, -0.43677347898483276, -1.0509741306304932, -0.4987673759460449, -0.07137534767389297, -2.639686107635498, 0.9454784393310547, -1.7765456438064575, 0.8596646785736084, 1.126631498336792, -0.5106083154678345, -0.44357678294181824, -0.9958270192146301, -1.8304829597473145, -1.4629961252212524, -0.20733202993869781, 0.8100247383117676, 0.13952408730983734, -1.2643884420394897, 0.7677369117736816, 0.520555317401886, 1.1303008794784546, -0.029671350494027138, 0.9163188934326172, 2.193613052368164, 1.7140898704528809, 1.2626360654830933, 1.2305337190628052, 0.3714805841445923, 1.8394039869308472, -0.3278053402900696, -0.5003227591514587, 0.15585052967071533, -0.3437369763851166, -1.0329766273498535, -0.8723035454750061, 0.2562001943588257, 0.4674362242221832, 2.7400801181793213, 1.0468546152114868, 0.21173328161239624, -1.5344207286834717, 0.25061720609664917, -0.47049567103385925, -0.9653546810150146, -0.3987670838832855, -1.0462554693222046, -1.4102150201797485, -0.38253119587898254, -1.5399898290634155, -0.33390921354293823, -0.12887848913669586, -0.4171822965145111, -1.4171727895736694, 0.7327983379364014, -0.764813244342804, 0.616349458694458, 0.2941023111343384, 0.7970535755157471, 0.3693029582500458, 0.4203697144985199, 0.031603071838617325, -0.8587568402290344, 1.5985854864120483, -0.23630845546722412, -0.6289480328559875, -0.4997069835662842, -1.461098313331604, -0.5849454998970032, 0.8482699394226074, -0.429024338722229, 0.5412055850028992, -0.7801675796508789, -0.2119888961315155, 0.6980926990509033, -1.4089854955673218, 0.06417419761419296, -0.48498648405075073, 0.5559074878692627, -1.0381828546524048, 0.3636927306652069, 0.7804534435272217, -0.06141467019915581, -0.12377844005823135, 0.026461279019713402, 0.48404133319854736, -0.4269842207431793, 0.87244713306427, 0.3109448552131653, 0.3491778075695038, -2.0751304626464844, 1.1623896360397339, 0.15486139059066772, -0.4495513141155243, 0.38749173283576965, 0.2903445065021515, -0.00336350011639297, 0.6508092284202576, -0.4487505555152893, -0.6212066411972046, 0.3938520848751068, 1.788194179534912, -2.4168412685394287, -0.5831082463264465, -0.18545366823673248, -0.9771209359169006, -0.0682697519659996, 0.492695689201355, -0.6044516563415527, -1.1089372634887695, 0.4905165731906891, 1.4575506448745728, -1.934900164604187, 0.23315712809562683, 2.737069606781006, -0.43775060772895813, -0.9408378005027771, -0.7763295769691467, 0.3277234137058258, -0.3314298391342163, -1.619732141494751, 1.457872748374939, 0.5892742276191711, 0.018039459362626076, -0.8220813274383545, 1.2313599586486816, 1.8982888460159302, -0.7355995774269104, 0.6085594892501831, 0.77121502161026, -0.6140694618225098, -1.2021280527114868, 1.1722280979156494, 0.11175677180290222, -0.2127235233783722, -0.7203893065452576, 0.2536499798297882, -0.36885127425193787, 1.320130467414856, -0.3303062319755554, 0.4358873963356018, 0.5653366446495056, -1.5866432189941406, -2.603973865509033, 0.1453542858362198, -2.322669267654419, -0.20526351034641266, -0.24837255477905273, -0.15365365147590637, 0.22006644308567047, -1.054311752319336, 1.1373343467712402, 2.0551135540008545, -0.574508547782898, 1.8683260679244995, -1.1393070220947266, 1.0190266370773315, 0.31017062067985535, 0.5568709373474121, -1.5836248397827148, 0.00822924543172121, -0.4512803554534912, 1.0152851343154907, 0.24523349106311798, -0.4276275336742401, 0.30203136801719666, 0.6915526986122131, 0.08645758032798767, 0.35320302844047546, -0.5721760988235474, -0.24971798062324524, -1.3433771133422852, 0.778765857219696, -0.7111753225326538, -0.39114588499069214, -0.48018184304237366, -0.868621289730072, -0.36499762535095215, 0.37463727593421936, 1.1928644180297852, 0.21741634607315063, 1.4160747528076172, 0.2717008590698242, 0.011788981035351753, -0.1449436992406845, -0.5944948792457581, 0.9392139911651611, 0.339277446269989, -0.5461215972900391, 1.941549301147461, 1.7634260654449463, 0.3217456340789795, 0.7008024454116821, 0.26622915267944336, 0.34579601883888245, 0.8904508352279663, -1.6629425287246704, 1.5198416709899902, 0.8422787189483643, -0.36924660205841064, -0.37334567308425903, 0.6024665236473083, -1.501550555229187, 0.6455656290054321, -0.7902035117149353, -1.0641241073608398, -0.5135168433189392, 2.8965508937835693, 1.460849642753601, 1.1290570497512817, 0.042031265795230865, -1.1883387565612793, 1.5240248441696167, 0.3085636496543884, -1.6386760473251343, -0.2523054778575897, 1.596897006034851, -0.31105366349220276, -0.19628983736038208, -1.0314695835113525, 0.6715629696846008, 1.1315263509750366, -0.7513697147369385, 1.8517019748687744, -0.8985390663146973, -1.0607479810714722, -0.44654738903045654, -1.1876959800720215, -0.44534632563591003, -0.3509858250617981, -0.6400534510612488, 1.5963298082351685, -2.2711708545684814, -0.7553130984306335, 1.487589955329895, -1.6293935775756836, -0.36621275544166565, -2.5659382343292236, 0.836674153804779, -0.07102959603071213, 0.4513159990310669, 1.319038987159729, 0.21108129620552063, -0.555963933467865, 0.6371204853057861, -0.05854089558124542, -0.1323789358139038, -0.3955542743206024, -0.8006978034973145, 0.10301387310028076, 0.6267832517623901, -1.3052539825439453, -0.8364052772521973, -1.0414375066757202, 0.09015286713838577, 0.2461976259946823, 0.7839521765708923, 2.1990015506744385, -1.1352146863937378, -0.6295064687728882, 0.2515489161014557, 0.7646804451942444, 1.2722662687301636, 0.6531565189361572, -2.7918200492858887, 0.1552601456642151, -0.5626052021980286, 2.55896258354187, -0.7064609527587891, -1.1183967590332031, 1.291486382484436, -0.664318859577179, 0.7982432246208191, -0.5932600498199463, -0.5011079907417297, 1.2982380390167236, -1.1721991300582886, 0.6938914060592651, -0.6287970542907715, 0.4662019908428192, -0.3113473951816559, -0.6705760359764099, -0.4790393114089966, -0.4154509902000427, -0.15901294350624084, 0.7598133087158203, 0.022724399343132973, 1.3171881437301636, -0.5459511280059814, -0.8499432802200317, -1.132186770439148, 1.4997726678848267, 1.7482043504714966], [-0.6547475457191467, 1.0609298944473267, -0.5403397083282471, -0.2659735083580017, 0.5962584614753723, -0.8585357069969177, 0.8920462727546692, -0.5657343864440918, 0.1122848242521286, -0.3012920618057251, 0.7714498043060303, -0.2244539111852646, -0.3491518795490265, -0.5799925327301025, 0.5741101503372192, 0.45919281244277954, 0.4820769429206848, -0.17443875968456268, 0.07675714790821075, -0.05092678219079971, 0.5953398942947388, -2.4461262226104736, 0.6792337894439697, -2.2306501865386963, 0.02530777081847191, -0.9233281016349792, 0.649039626121521, 1.047022819519043, 0.6420233249664307, 1.8114376068115234, 0.5913655161857605, 0.06312959641218185, 0.1397695094347, 1.6538468599319458, 1.9708744287490845, -1.1188000440597534, -0.4035181403160095, -1.1683762073516846, 0.4805923104286194, 2.324882745742798, -0.48912569880485535, -1.9995051622390747, -0.4484897553920746, -0.2562566101551056, -0.42999809980392456, -0.6074813008308411, -0.30948662757873535, -1.348652958869934, -0.5684440732002258, -0.798339307308197, -1.317825198173523, 0.9168205261230469, 0.6240137815475464, -0.7581238746643066, 1.1711443662643433, 1.3815007209777832, -0.8047688007354736, -1.626872181892395, 0.3767646849155426, -1.3390158414840698, -1.2381619215011597, -0.4037613868713379, -0.2758319675922394, 0.12459423393011093, 0.4012949466705322, -0.276352196931839, 0.29154932498931885, -0.9587371945381165, -0.6754692792892456, 0.7239517569541931, 0.5745200514793396, -1.8541136980056763, -0.5316506028175354, -0.7677192687988281, -0.8753967881202698, -0.5467109084129333, -1.2071912288665771, 0.08675391972064972, 0.6058796048164368, -0.4827079474925995, 1.2095298767089844, 0.22155535221099854, 1.6557352542877197, 0.5641718506813049, -0.27906182408332825, -0.07967213541269302, -1.0439190864562988, -0.002944840118288994, 1.140179991722107, -0.8612323999404907, 0.6484136581420898, 1.0853347778320312, -1.6127333641052246, -0.5349969863891602, 0.25173401832580566, -0.9050861597061157, 0.6641849279403687, 0.22236159443855286, 0.4086911678314209, 1.7227108478546143, 0.9894124269485474, 0.9343738555908203, 0.01673663966357708, 1.2469635009765625, -0.12163455039262772, 1.799124836921692, 2.436981201171875, 0.10489250719547272, 0.692390501499176, 0.2399042546749115, -0.28015226125717163, -2.021796703338623, -0.08522374927997589, 0.3169405162334442, 0.6528307795524597, -1.4264949560165405, -0.29698124527931213, -0.2239704430103302, 0.5039039254188538, -0.7673731446266174, -1.1229268312454224, -0.4041547477245331, -0.5351099371910095, -1.7888870239257812, 0.09247654676437378, 0.4500987231731415, 0.30266398191452026, 1.8246374130249023, -0.48864826560020447, 0.62850022315979, 0.6063310503959656, 0.04201539233326912, -0.25829413533210754, -0.1895723044872284, 1.4306219816207886, -0.27878841757774353, 1.3453121185302734, 0.92973393201828, 1.332669734954834, -1.6789401769638062, -0.08154436200857162, -0.8679249286651611, 0.016695071011781693, -2.258587121963501, -0.3409802317619324, 0.25013941526412964, -1.5835530757904053, 0.027143388986587524, -0.8448503017425537, 1.1462771892547607, 0.27503958344459534, -0.8491935133934021, 0.1063774898648262, 1.7838411331176758, -1.407415509223938, -1.8882577419281006, 1.0087946653366089, 1.5326489210128784, 0.7454683780670166, 0.49355435371398926, 0.8352893590927124, 1.446597933769226, -1.8141252994537354, 0.3673764169216156, -0.4750041961669922, 0.11010314524173737, -0.2747582793235779, 0.12893109023571014, -0.48496848344802856, -0.00788507703691721, 1.4460734128952026, 0.5458351969718933, -0.2069781869649887, -1.35188627243042, 0.6295419931411743, 0.8783957362174988, 0.8767526149749756, -0.32182174921035767, -1.3558986186981201, 0.4962446987628937, -1.8795562982559204, -0.5621033310890198, 0.6037599444389343, 0.6199238300323486, 0.43649017810821533, -0.5403695702552795, -0.7122650742530823, -1.0351264476776123, -2.0734593868255615, 0.1784823089838028, -1.3656902313232422, 0.30273839831352234, 1.405603051185608, 1.3241384029388428, 0.21014609932899475, -0.8131527304649353, 0.988198459148407, -0.4524916112422943, 1.0201823711395264, -0.3864854872226715, 1.8972021341323853, -0.4889044165611267, 1.8197021484375, 1.3345211744308472, -2.1299707889556885, 0.02731435000896454, 1.1731524467468262, 0.11725128442049026, -0.05310327559709549, 0.10190083831548691, 0.15073291957378387, -0.14478814601898193, 0.2833572328090668, -0.945366621017456, -1.0191636085510254, 0.13321807980537415, 0.37932145595550537, 0.9815230965614319, 0.13247495889663696, -0.10692761838436127, -0.0785503014922142, 0.5488104820251465, -0.33048132061958313, -1.3002166748046875, 0.6001008749008179, -1.0718392133712769, -0.7538759708404541, 0.310001015663147, 1.2124642133712769, 0.41642555594444275, 0.41389232873916626, -0.7891073822975159, -0.18875619769096375, -0.6209863424301147, -0.7689125537872314, -0.48882248997688293, -0.38228848576545715, -0.7555440068244934, 1.57706618309021, 0.4681296944618225, 0.21382002532482147, 0.43274354934692383, -0.030436363071203232, -3.0190515518188477, -1.0253571271896362, 1.1072756052017212, -0.10474137961864471, -0.1037856936454773, 1.200185775756836, -0.5406899452209473, -0.8516013026237488, 0.12218128889799118, -0.8154129385948181, 0.8126399517059326, -2.501383066177368, -0.4158274531364441, -1.297873854637146, -0.23066100478172302, -0.0400942862033844, 1.0214660167694092, 0.47452011704444885, 1.6605068445205688, -0.2314121127128601, 0.8440597057342529, -1.4764442443847656, 1.7494657039642334, -0.17603260278701782, 1.348613977432251, 0.2515452802181244, -1.432517409324646, 0.07142109423875809, 0.48988622426986694, 0.40288472175598145, -0.438725084066391, 1.339888572692871, -0.5107765793800354, 1.7268964052200317, 0.11884734779596329, 0.8660082817077637, 1.622382640838623, -0.6507785320281982, -0.9966859817504883, -1.6917167901992798, 0.7122698426246643, -0.017773548141121864, 0.8294127583503723, -2.7431182861328125, -0.8317066431045532, 0.05225319042801857, 1.9538872241973877, -0.6664633750915527, -0.13739338517189026, -0.847794771194458, 0.6052072048187256, -0.44178396463394165, -0.9739946126937866, -0.20490439236164093, -0.24192696809768677, 0.41710585355758667, 1.9031789302825928, 0.25565579533576965, 0.1146605983376503, 0.5409925580024719, -0.8265617489814758, 0.5157520174980164, -1.2121760845184326, -1.3797820806503296, -0.42420223355293274, 0.5810847282409668, -1.6163339614868164, 0.16235533356666565, 0.33952924609184265, -0.05769634246826172, -0.431617796421051, -1.806020975112915, 1.5039217472076416, 0.6874573230743408, -0.16291609406471252, -1.490486741065979, 0.3758622109889984, -0.8258845806121826, 1.9806429147720337, 0.6034107208251953, -0.6319431662559509, -3.230046510696411, -0.17591531574726105, -0.9111583232879639, -0.21762125194072723, 0.1591484397649765, 0.5405108332633972, -1.3523355722427368, 0.732927680015564, 0.11912905424833298, -1.1227811574935913, 0.4371492862701416, -1.3200562000274658, -0.46212369203567505, -1.5471246242523193, 0.9349250793457031, -0.00594687694683671, -0.9116335511207581, 0.8570047616958618, -0.6363033652305603, -0.29338762164115906, -0.5944366455078125, 0.5540822148323059, -0.8912266492843628, -0.754914402961731, 0.31288355588912964, -1.3800206184387207, -0.6220617294311523, -0.3786751627922058, -0.09471700340509415, 0.26754531264305115, 2.156170606613159, -0.23589091002941132, 0.059064097702503204, 0.17181505262851715, -1.0822833776474, -0.8881892561912537, 0.0683128759264946, -0.504818856716156, 0.5365815162658691, 2.0444202423095703, -0.9326083660125732, -1.0866951942443848, 0.6121909618377686, -0.10645713657140732, -1.0721087455749512, -1.6844110488891602, 0.3457576036453247, -2.047563314437866, -0.36428341269493103, -1.110876202583313, 0.3336934745311737, -1.5747710466384888, 0.3271014988422394, 1.161275863647461, 0.2728647291660309, 0.718178927898407, 0.3422826826572418, -0.9144077301025391, -2.9428882598876953, -2.435772657394409, -0.6595875024795532, -0.7461927533149719, 0.08719553053379059, -0.4881565570831299, 0.7309613823890686, -2.7044837474823, 0.0791993960738182, 0.8325942754745483, 0.7642599940299988, -0.2713286578655243, 0.5488659739494324, -0.3397231698036194, 0.9894027709960938, -0.2505899965763092, -0.902206540107727, 1.378361463546753, 1.671520471572876, 0.7496379017829895, 0.2388114631175995, 0.38751891255378723, 0.2560935318470001, -0.0705057755112648, -0.5456016063690186, 1.4667295217514038, 0.45517000555992126, -1.2058111429214478, 1.746748447418213, -0.3017757534980774, -0.08031079918146133, -1.6845149993896484, 0.8346534967422485, -0.022309133782982826, 0.02438393421471119, -1.0947973728179932, 0.9464772343635559, -2.122549057006836, 1.3222509622573853, 0.5201769471168518, -0.17669297754764557, 0.606785774230957, -0.014184524305164814, 1.2692843675613403, 1.3710662126541138, 2.744671583175659, 0.4820321500301361, -2.4820308685302734, -1.510267972946167, 0.7484984993934631, -0.5224688649177551, -0.9798474311828613, 0.6159111261367798, -0.8462737202644348, -0.8815096616744995, 0.11954671144485474, 1.5648770332336426, -0.13063326478004456, -1.170238733291626, -0.897444486618042, -1.4611202478408813, 1.9549123048782349, 0.2411573827266693, 0.0626596137881279, -0.014272188767790794, 0.7669314742088318, 1.290403127670288, 0.357941210269928, 1.5772823095321655, 0.7509616017341614, -0.7756772041320801, 1.1887503862380981, -0.9820055961608887, 0.06550073623657227, -0.05337440222501755, 0.4541681706905365, 1.8554593324661255, 2.2943434715270996, -0.9592976570129395, -0.23292765021324158, -1.7167471647262573, -0.5219756364822388, 0.46420854330062866, 1.4580219984054565, 1.6716080904006958, 0.1699499636888504, 1.200592041015625, -0.3979339599609375, 0.7586926221847534, -0.03137839585542679, 1.2893714904785156, 0.8636103272438049, 2.8570055961608887, 0.006033053155988455, -0.7172144055366516, -1.520930528640747, 0.7107237577438354, -0.10300113260746002, -1.7117414474487305, -1.5913171768188477, 1.6257209777832031, 1.1030465364456177, -1.1568326950073242, 0.5620722770690918, 1.7644140720367432, -0.44992712140083313, 0.9950082898139954, 0.7138838768005371, -0.3706279993057251, 0.06854815781116486, -0.5643925666809082, -0.9765990972518921, -0.11892513930797577, 0.4622911214828491, 1.053259253501892, -0.2705935537815094, 1.5996557474136353, 0.8008298277854919, 0.6962646842002869, -0.8742421269416809, 2.055013418197632, 0.7838121652603149, -1.7812985181808472, -1.0951999425888062, -0.03705381602048874, -0.08934490382671356, -0.5240772366523743, -0.5432465076446533, 1.3318918943405151, 0.8048663139343262, -1.193625807762146, -0.211096853017807, 2.3730993270874023, 0.1948542296886444, -1.0949617624282837, -2.250617265701294, 0.11816443502902985, 0.24633251130580902, 0.36396393179893494, -0.5445042848587036, -0.010907582007348537, 0.3681505620479584, 0.28573668003082275, 0.6817429661750793, 1.8827722072601318, 0.21134816110134125, -1.655701994895935, 0.28279536962509155, 1.1718817949295044, -0.5983641147613525, 2.4434428215026855, -0.14780668914318085, -1.1370044946670532, 0.2911318838596344, 0.07789397239685059, -0.3148826062679291, 0.6297441720962524, -1.0888432264328003, -1.207364797592163, 0.047389350831508636, 0.6902941465377808, 0.8670748472213745, -0.22053389251232147, -1.5164145231246948, -0.04135601595044136, -0.8535945415496826, -1.822052240371704, 0.7835003733634949, -0.8182433843612671, 0.9489713311195374, 1.719997763633728, -0.893169641494751, 0.3623441159725189, 0.4701833426952362, 1.2693428993225098, -0.975226879119873, 0.20079204440116882, 1.7214709520339966, -1.1105129718780518, 1.0070561170578003, -0.44231116771698, 2.270294666290283, 0.39440134167671204, 1.0462528467178345, 0.18749284744262695, 0.3884701728820801, -1.2889454364776611, -2.531511068344116, 0.898353636264801, 0.7941461205482483, 1.55246102809906, 0.930349588394165, 0.0284495260566473, -0.4662797451019287, 1.3109179735183716, 0.9827528595924377, 1.7298201322555542, 1.6850306987762451, 1.401172399520874, -0.04465613141655922, 0.935492992401123, 2.7748608589172363, -1.863879919052124, 0.058480583131313324, -0.16122032701969147, -1.0602458715438843, -0.5095160007476807, 0.2601846158504486, 0.5375137329101562, -1.1172401905059814, -0.7512112259864807, 2.5738890171051025, -0.36046063899993896, 1.4612725973129272, -0.825336754322052, -0.11746539920568466, -0.13171859085559845, 1.5580235719680786, 1.2913503646850586, 1.3914297819137573, -1.0731785297393799, -0.1756487488746643, 1.0261327028274536, 1.4050650596618652, -0.10328598320484161, -0.5755791664123535, -0.7949486374855042, 0.5562536120414734, 0.03891599178314209, 0.03864045441150665, -0.9159530997276306, -1.7406030893325806, 0.8640550374984741, -0.8710504770278931, 0.24856038391590118, 0.729193925857544, 0.6878985166549683, 2.2376718521118164, -0.15953561663627625, 0.8698701858520508, -0.2162773460149765, -1.1971005201339722, -0.9202012419700623, 0.6373119950294495, 1.8955680131912231, 1.9164888858795166, -0.21254120767116547, 1.793750286102295, -2.10410737991333, -0.19427280128002167, 1.8817634582519531, -0.22540703415870667, -1.002519130706787, 0.3405117094516754, -0.8420965075492859, 1.4981449842453003, 1.04754638671875, -1.3184635639190674, -0.021694611757993698, -1.4254529476165771, 1.4278408288955688, -0.6189022064208984, 0.2413346916437149, 2.079585552215576, -0.1458083987236023, 0.5522559285163879, -0.33295363187789917, -0.8998681306838989, -1.8102606534957886, -0.10133378952741623, -1.710432767868042, 0.6716140508651733, 1.1329669952392578, 1.2185769081115723, -0.8516470193862915, -0.5827186703681946, 1.8234612941741943, -0.35471659898757935, 1.5169724225997925, 0.32566946744918823, 0.8894290924072266, 1.6112021207809448, -0.5914004445075989, -0.5941817164421082, -1.3596231937408447, 0.9859166145324707, -1.1063027381896973, -0.21017540991306305, 0.9510321021080017, -0.520661473274231, -0.11487912386655807, -0.02618255279958248, 1.4737499952316284, -1.7408478260040283, 1.2581849098205566, 0.3982974886894226, 0.2231212705373764, 0.2188527137041092, -0.33931565284729004, 0.25752726197242737, 1.332866907119751, -0.4296961724758148, -0.01435389555990696, -1.326145887374878, 0.5996822714805603, 1.135791301727295, -0.4617825746536255, -0.4183104634284973, 0.5055651664733887, -1.7326279878616333, 1.1211034059524536, 0.017539843916893005, -0.2532101273536682, -0.4731108844280243, -1.1126840114593506, 0.2565430700778961, 0.45301952958106995, -2.3758738040924072, 0.1293896585702896, -1.1873595714569092, -0.4219684898853302, -2.046293020248413, -0.2921920716762543, 1.2660174369812012, -0.2120818793773651, -0.5857663154602051, -0.4414655864238739, -0.6389148831367493, -1.748193383216858, -1.481661319732666, -1.477068543434143, -0.1956159919500351, 0.9080036282539368, -1.058117151260376, 0.20816373825073242, 0.29184210300445557, -1.2928885221481323, 1.2671061754226685, -0.6807606220245361, 1.2841821908950806, -1.3786697387695312, -0.024701371788978577, -0.7458561658859253, -1.4878137111663818, -0.3556711971759796, -1.3644622564315796, -0.5327361226081848, 1.5979533195495605, -1.0802497863769531, 0.15550106763839722, -0.5917115211486816, -1.1694058179855347, 0.6594216823577881, 0.058143243193626404, -0.7749154567718506, -0.8790615200996399, -0.7002843618392944, -0.2877180576324463, -0.062168437987565994, 1.1829453706741333, -1.3866280317306519, -0.25328269600868225, 0.9752150774002075, 0.6001080274581909, -1.0978176593780518, 1.9325687885284424, -0.9838199615478516, 0.36604395508766174, 0.44139963388442993, -1.1500502824783325, 0.5376094579696655, 0.282253623008728, -1.3917549848556519, -0.42992162704467773, -2.3321824073791504, 0.42745038866996765, 0.31400588154792786, 1.0769867897033691, -0.002890544245019555, 1.6717935800552368, -0.5493699908256531, -0.3231225907802582, 0.12328745424747467, 0.37615248560905457, 0.3051726818084717, -0.646502673625946, 0.9948546886444092, 0.8576215505599976, 0.16329997777938843, -2.337888717651367, -0.26439976692199707, 0.32836198806762695, -0.9324831366539001, 0.06400974839925766, -0.3458906412124634, 3.102306365966797, 0.6380816102027893, 0.837733268737793, -0.42276692390441895, 2.1206068992614746, -0.49329686164855957, -0.7842307090759277, 0.9544544219970703, -1.6516975164413452, 0.25300511717796326, 0.6756044030189514, -1.3508673906326294, 0.8676527142524719, 1.6485249996185303, -1.5570448637008667, -0.41452357172966003, 0.11940772086381912, 0.5042057037353516, -1.098484754562378, -0.486203134059906, 0.07049315422773361, -0.2821611762046814, 1.844300389289856, -0.9036076068878174, -1.5623323917388916, -0.2670658230781555, 1.7868555784225464, -0.8309748768806458, 0.562883198261261, 1.4759279489517212, 0.08887691050767899, -1.0989460945129395, -2.384425163269043, 1.2585504055023193, -0.812885046005249, -0.008477275259792805, -2.444092273712158, 0.45568180084228516, 1.4403076171875, 1.4993939399719238, -0.5215750932693481, 0.1441546529531479, 0.9113926291465759, 0.9953590035438538, -0.033621303737163544, 0.08018726110458374, 0.4381524324417114, -2.6071982383728027, -1.0417948961257935, -1.3252387046813965, 0.9392897486686707, -0.18957816064357758, -0.2440352737903595, -1.5888402462005615, -1.2386904954910278, 0.21699218451976776, -3.12630033493042, -0.1250952184200287, 0.9030758738517761, 0.18512293696403503, -0.9446293711662292, 1.0680663585662842, -0.2858092784881592, 0.12024889886379242, 1.0887478590011597, -1.0698661804199219, -0.25052329897880554, -0.6222876310348511, 1.6588910818099976, 0.3888920545578003, 2.289376974105835, 1.4816761016845703, 0.022453246638178825, 1.1875801086425781, 1.1601423025131226, -0.153237983584404, 1.043562412261963, -0.277108371257782, -2.0168299674987793, -0.5720380544662476, -0.6967962384223938, -0.9543178677558899, 1.3353098630905151, 1.0074150562286377, 2.050513982772827, 0.3020593225955963, -0.05942509323358536, 0.866108775138855, 0.6302253007888794, -0.5715867280960083, -1.384174108505249, -0.6817476153373718, -0.6330870985984802, 0.05938582122325897, -0.37893179059028625, 0.35858026146888733, -0.5949729084968567, -0.7760034799575806, 1.3333057165145874, 1.742897629737854, 0.3218880891799927, -1.642702341079712, 1.026330590248108, -0.25149568915367126, 2.2736170291900635, 0.22604848444461823, 0.6507582068443298, -0.6303489804267883, 0.9213649034500122, -0.8733713030815125, -0.28759312629699707, -0.6984574198722839, -0.5373167991638184, -0.8914382457733154, 0.858514130115509, 0.7643234133720398, 0.0074948640540242195, 0.3063201904296875, -0.06327691674232483, -0.958705723285675, -1.4067343473434448, -0.6499465703964233, 1.3213388919830322, 0.28348198533058167, 0.3179522156715393, 0.32929643988609314, 0.49417296051979065, -0.4076637923717499, -0.3271445333957672, 0.768585205078125, -1.288715124130249, 1.050690770149231, 0.6516738533973694, 0.9781944751739502, 2.154690980911255, -1.9221197366714478, -0.4334999620914459, 0.9191218018531799, 0.6102591753005981, 2.223799228668213, -1.2180625200271606, -0.2920565903186798, -0.6254577040672302, 0.07992289960384369, 1.1993660926818848, 1.9381884336471558, 1.4312626123428345, 1.7112971544265747, -0.5248342752456665, -1.8197883367538452, 0.12398475408554077, -1.0916413068771362, -0.2659960687160492, -1.890230655670166, 1.2590363025665283, -0.39509186148643494, -0.606824517250061, -0.31333228945732117, -0.2316099852323532, 0.16260792315006256, 0.3506848216056824, 0.2794860601425171, 0.9192644953727722, -0.7048985362052917, 0.707289457321167, 0.42780110239982605, 1.357344388961792, -2.0086669921875, 0.2767072916030884, 0.6308318972587585, 0.3137436807155609, 0.24404461681842804, 0.3899371325969696, -0.015362033620476723, 0.7350630164146423, -0.04952627792954445, -0.416907399892807, 0.43839746713638306, 0.06535884737968445, 0.020258191972970963, -2.3210389614105225, 0.955527126789093, 0.21453239023685455, -0.8260916471481323, 1.5594825744628906, 0.565995454788208, -0.6414312720298767, -0.20895424485206604, -1.2454743385314941, 0.4384019672870636, -0.3803692162036896, -1.2947171926498413, 0.2576098144054413, -0.6212818622589111, -2.308845281600952, 0.9732663035392761, -1.1955863237380981, 0.5902194380760193, 0.6438506841659546, 1.2305099964141846, 0.6525393724441528, 0.183531254529953, -1.9923487901687622, -1.7737244367599487, -1.68267023563385, -2.0374937057495117, -0.7874827980995178, -1.1435058116912842, -1.3245426416397095, -1.6891010999679565, -1.0064592361450195, -0.3654992878437042, -1.0838836431503296, -0.6292558312416077, -1.3767746686935425, 0.34519773721694946, 0.3712174594402313, -0.07838230580091476, -2.2894465923309326, 0.1286209523677826, -1.0614237785339355, -0.8626571893692017, -2.3125813007354736, -0.02953203022480011, 0.6696529388427734, 1.0561023950576782, -0.15145201981067657, -0.21751151978969574, -0.5430793166160583, -0.14166076481342316, -1.1450875997543335, -0.5241629481315613, -0.6723490357398987, -1.9545111656188965, -0.3869225084781647, 0.09746643900871277, -0.28639745712280273, -0.8371102213859558, 0.11428233236074448, -0.3550599217414856, 0.6549363732337952, 0.6535177826881409, 1.3637615442276, 0.9904863834381104, 0.8592748045921326, -0.059832170605659485, -1.4705461263656616], [-0.1326347142457962, -0.9933157563209534, -0.20266950130462646, 0.581257700920105, -1.381113886833191, 0.2763846516609192, 0.1360158622264862, -1.6683226823806763, -1.0778164863586426, -0.359500527381897, 0.908042311668396, -0.15821655094623566, -0.7369927167892456, -0.7108126878738403, 0.9569486975669861, 0.8910977840423584, -0.864198625087738, -1.8976259231567383, -0.5250091552734375, -0.08107835054397583, 0.7494518160820007, 1.4209877252578735, -1.2369475364685059, 0.3748042583465576, -0.2776080369949341, -1.8792988061904907, -0.4375493824481964, 0.048937682062387466, 0.1127639189362526, 0.8568938374519348, 0.6972331404685974, 0.13828854262828827, -0.9133880734443665, 0.32656121253967285, -1.1074436902999878, 0.4582787752151489, 1.038108468055725, 1.1375112533569336, -0.9365124106407166, -2.301305055618286, 0.22367922961711884, -1.2213406562805176, 0.2010316252708435, -0.7019588351249695, 0.033558789640665054, -1.720357060432434, 0.7769100069999695, -0.15663115680217743, -0.29968881607055664, 0.06577297300100327, 0.907402753829956, -1.3965470790863037, 1.449594497680664, -0.4047943651676178, -0.8199842572212219, 1.5814052820205688, -0.9732933044433594, 2.00454044342041, 0.34152060747146606, 0.4953637719154358, 1.6389153003692627, 0.9364057779312134, 0.13653357326984406, -1.7554450035095215, -0.3999329209327698, -1.2159384489059448, -0.7448284029960632, 0.9173254370689392, -0.5157456398010254, 0.8291832208633423, 0.9110057353973389, 0.8250297904014587, -0.8744283318519592, 0.8083570599555969, -0.6678745150566101, 0.7513278126716614, 0.7559487223625183, 0.3978087902069092, 0.4816860258579254, -0.06316786259412766, -0.6023319959640503, -1.5647451877593994, -0.4509914815425873, -1.144697666168213, 1.1614007949829102, -0.650353193283081, -0.5905905365943909, -0.3009001910686493, -0.3282954692840576, 0.9021649360656738, -0.7355561256408691, -1.961738109588623, 0.8679843544960022, -0.5387915968894958, 0.38923951983451843, -1.2112196683883667, -0.4114035665988922, 2.023815155029297, 1.5325433015823364, 0.9390174150466919, 0.31180840730667114, -0.9702919721603394, 0.32462388277053833, 0.12202401459217072, -0.0006277764332480729, 0.7862882018089294, 0.3053845465183258, 0.2767769694328308, -0.26074153184890747, -2.2929725646972656, -0.4079226553440094, -1.244275450706482, -0.9920029640197754, -2.933095693588257, 0.5231603980064392, -2.1031346321105957, 0.9931933283805847, 0.15658174455165863, 0.9131348729133606, 0.29554271697998047, -0.7618858814239502, -1.074785828590393, -0.8712499737739563, 0.5729014277458191, -2.089776039123535, -0.08634182065725327, -1.194455862045288, 0.5872438549995422, 0.18747222423553467, 0.03691841661930084, 0.01764896884560585, 0.13182441890239716, -0.27027273178100586, 0.9960172772407532, -1.2826424837112427, 1.3254585266113281, -0.10242467373609543, -1.077546238899231, -1.4383273124694824, -0.9727341532707214, 0.31545451283454895, 1.03139328956604, -0.8831172585487366, 0.012353690341114998, 0.01905052363872528, -0.2757311165332794, 1.3002934455871582, -0.30762016773223877, -0.7103997468948364, -1.8605456352233887, 0.25927793979644775, -1.011680245399475, -0.20949380099773407, 0.7976972460746765, -0.4798295497894287, -0.7142173051834106, -0.1820276975631714, -2.778205633163452, 0.5070626139640808, -1.476039171218872, 1.1609439849853516, 2.044987678527832, -0.3564266860485077, -0.41468343138694763, -0.2913673222064972, 0.0076406472362577915, -1.8848224878311157, -0.42406588792800903, 1.3937218189239502, -0.3907102942466736, -1.6967442035675049, 1.0530850887298584, -0.18972036242485046, 0.9658655524253845, -1.160631537437439, -2.937649965286255, 1.6102691888809204, -0.08219633251428604, 1.2675212621688843, -0.5132765173912048, -1.4021013975143433, -0.9517051577568054, 0.04512620344758034, -0.4388125240802765, 0.5852999687194824, 0.01102663017809391, -0.7118918299674988, 0.5327358245849609, -2.435288429260254, 1.3488789796829224, -1.1460984945297241, 0.9667487740516663, 0.7357400059700012, 0.7181349992752075, 0.26963314414024353, 0.9819132685661316, -1.2779486179351807, 1.1971789598464966, -1.3186633586883545, -0.30903729796409607, -0.05260895565152168, -0.16190975904464722, -1.5203784704208374, -0.7685256004333496, -0.6218374371528625, 0.2549690008163452, 1.4529367685317993, -1.0418399572372437, -1.4098459482192993, 0.9787169694900513, 0.14688847959041595, -0.8480359315872192, 0.6561294198036194, 0.39813271164894104, -0.042897358536720276, -0.6021369695663452, -0.6321802139282227, -0.02286483347415924, 0.4098130464553833, -0.3991650640964508, 0.6875076293945312, 1.2787725925445557, 1.1303061246871948, -0.7615044713020325, -0.662836492061615, 0.813693106174469, -0.1859617829322815, -0.36586496233940125, -1.23560631275177, -0.8305461406707764, 0.4849533438682556, -0.3938505947589874, -1.4288678169250488, 0.1753736436367035, -0.23350897431373596, -0.41908568143844604, 1.5578932762145996, 0.6536356210708618, 0.14689049124717712, -0.06715414673089981, -1.0596039295196533, 0.18116658926010132, 1.6356427669525146, -0.1706220954656601, 0.9227731823921204, 1.5204211473464966, 1.394879937171936, -0.4923005700111389, 0.7629730105400085, -1.302422046661377, -0.6988946795463562, 1.0809211730957031, -2.207451581954956, 0.6901854276657104, -0.1624331921339035, 0.3178047835826874, 0.08011801540851593, -1.4424974918365479, -0.010227812454104424, 0.4193595051765442, 0.12348803132772446, -1.0988945960998535, 0.24045318365097046, -2.309737205505371, -0.8068416714668274, -1.4582180976867676, 0.8847833871841431, -0.9959174990653992, -1.237827181816101, -1.1303186416625977, -1.0585880279541016, 1.5497760772705078, 0.2551996409893036, 0.14920228719711304, -1.0236493349075317, 0.04497063532471657, 0.3030813932418823, -0.6092702746391296, 1.4980778694152832, -0.9436851739883423, -0.04110772907733917, 1.0377106666564941, 0.9227617383003235, -1.4380300045013428, -1.2137424945831299, -0.746963620185852, 0.09015001356601715, 0.3814482092857361, 0.6425983309745789, -0.8884313702583313, -0.5125536918640137, 2.211458921432495, 0.4700356125831604, 0.10268879681825638, -2.348095178604126, -0.07375387102365494, -1.3360447883605957, -1.0228413343429565, 0.6590991616249084, 0.3199683427810669, -0.45689812302589417, -1.303632378578186, 0.6917744278907776, -1.6094725131988525, 1.4543477296829224, 1.3285138607025146, 1.0183974504470825, 0.2785239517688751, -0.339386910200119, 0.5990316271781921, -0.19663307070732117, 0.888367772102356, -1.6678715944290161, -0.11232597380876541, -0.37051331996917725, -0.18201498687267303, 1.2623859643936157, -0.21349498629570007, -0.10864869505167007, 0.32840895652770996, -0.7874798774719238, 0.3566669523715973, 0.0989062711596489, 0.384417325258255, 1.2332706451416016, 0.7686100006103516, 1.0828579664230347, 1.6978625059127808, -0.5478070974349976, -0.3801772892475128, -0.7274034023284912, -1.1382403373718262, 0.29082316160202026, 0.12520012259483337, -0.07539225369691849, -0.1788613498210907, -0.7988713383674622, -0.6520929932594299, 1.0456851720809937, 0.5478583574295044, -0.09583970159292221, 0.8939499258995056, 1.8103117942810059, 1.576700210571289, 0.19022931158542633, 0.11632026731967926, 0.20747454464435577, -2.2621071338653564, 1.8067777156829834, -0.4421444535255432, 2.215045690536499, 0.04261469468474388, 1.1828408241271973, 0.6358910799026489, -1.0653107166290283, 0.3025673031806946, -2.411062717437744, 0.13881680369377136, -0.021101132035255432, -0.18855607509613037, 1.395990252494812, 0.4639301300048828, -0.994598388671875, 1.2745124101638794, 0.19064298272132874, 0.16406458616256714, 0.41791632771492004, 0.07261157035827637, -0.7604158520698547, 1.052438497543335, -2.252591133117676, 1.092326283454895, 1.6932710409164429, -0.271465927362442, 1.692557692527771, -0.9363909959793091, -1.1319904327392578, 0.0019464666256681085, 0.11780419200658798, -0.3477167785167694, -0.05810746178030968, 1.1895638704299927, -0.0362655408680439, -0.3608722388744354, 2.2950751781463623, -0.868493914604187, -0.6381974220275879, -0.47040075063705444, -0.7293906211853027, -0.5478166341781616, 0.17215856909751892, 0.5576609969139099, -1.861591100692749, 1.0685834884643555, 0.5547104477882385, -0.01410536840558052, 2.7995877265930176, 0.7539600133895874, -0.6795329451560974, -0.6450682282447815, -0.12074653059244156, 1.3174612522125244, -0.02940542623400688, -0.19148266315460205, 1.0258318185806274, 0.1995995193719864, 1.2098621129989624, -0.4400723874568939, -0.5587680339813232, -1.4361323118209839, 0.4201672673225403, -0.031555354595184326, 0.9670430421829224, -1.1884511709213257, -0.22826331853866577, -0.9237051606178284, 0.6917375922203064, -0.02987378090620041, 0.7878641486167908, 0.047129012644290924, -0.017655139788985252, -0.11100517213344574, 0.6483100056648254, 1.1714205741882324, 0.9939535856246948, 1.328925609588623, 1.908559799194336, 0.034906644374132156, 0.5764002203941345, -0.27870991826057434, 0.3678288757801056, 0.44467833638191223, -0.13841576874256134, -1.1311194896697998, 0.2525542378425598, 0.36306771636009216, 0.32455047965049744, 1.3848131895065308, -0.1370924413204193, 0.5341573357582092, 0.9006957411766052, -0.4211964011192322, -0.6367531418800354, -0.8376269340515137, 0.9809379577636719, -0.5738496780395508, -1.3115917444229126, -0.9462472796440125, 0.9114605188369751, -1.3220590353012085, -1.0984712839126587, 0.6082100868225098, 0.4907331168651581, 0.16404989361763, -0.26723650097846985, -0.24355360865592957, 1.9387645721435547, 0.11181142926216125, -0.0029819367919117212, -1.4946039915084839, 2.022573232650757, 0.007092317100614309, -0.7926583886146545, 0.6102501153945923, 0.20491944253444672, 1.3739360570907593, -0.11477608978748322, 0.6776310205459595, -0.15511232614517212, -1.1027582883834839, -0.26970797777175903, -1.523640513420105, -0.20917156338691711, 1.0265188217163086, 0.2908991575241089, -0.13447576761245728, -0.07999615371227264, 1.7830201387405396, -2.172733783721924, 0.4730234742164612, -1.4525659084320068, 0.3126981258392334, -0.16107766330242157, -1.1074237823486328, 0.6777405738830566, 0.49329376220703125, -1.1299923658370972, 0.8908272385597229, 0.17235732078552246, -0.6158963441848755, 0.18475447595119476, 1.1594992876052856, 0.8090137243270874, 0.6677098274230957, 0.5880860686302185, -0.6938735842704773, 0.421753853559494, 0.033562514930963516, 0.8308455348014832, -0.5780367851257324, -1.0444246530532837, 1.1742992401123047, 0.2559430003166199, 0.7794250845909119, -1.0932646989822388, 0.39579126238822937, -0.6430395245552063, -1.4485254287719727, 0.9973880648612976, -1.1215907335281372, -0.05331281200051308, 1.6048321723937988, 0.34686899185180664, -0.9326767921447754, -1.3740336894989014, -0.8817682862281799, 1.150118112564087, -0.9416522979736328, 1.4081854820251465, 1.1104854345321655, 1.6499406099319458, 0.690371036529541, -0.3873237371444702, 1.6411198377609253, 0.2721917927265167, -0.6193565130233765, -0.19253048300743103, -0.24321594834327698, -0.533504068851471, 0.8171236515045166, -0.7368587255477905, 0.7518090605735779, 0.8403366804122925, 0.5872228145599365, -1.3998558521270752, -1.4136724472045898, -0.6953154802322388, 2.295980930328369, -1.8794622421264648, -0.6701898574829102, 0.7880542874336243, 0.9113838076591492, 1.1388739347457886, 0.45339882373809814, 0.6980350613594055, -0.6913391947746277, -0.34536460041999817, 1.1741251945495605, -0.5759900212287903, 0.1394149214029312, 0.21215535700321198, -0.5655670166015625, 0.6682286858558655, -0.6760289072990417, -0.8387637734413147, -0.9492838978767395, 1.1012517213821411, 0.11222045868635178, -0.7609477043151855, -0.748007595539093, -1.0806736946105957, 2.2056634426116943, 0.14132173359394073, -1.4537557363510132, -0.03464604169130325, -0.1260492354631424, -0.33293113112449646, -0.7776308655738831, -0.032247986644506454, 0.13264325261116028, -0.532468855381012, 0.3664993643760681, -1.1063491106033325, 0.46973758935928345, -0.137741819024086, 0.8630566000938416, -1.2565655708312988, 0.590897262096405, 1.8367668390274048, -3.0835955142974854, 1.537691593170166, -1.3655364513397217, -0.4775301516056061, -1.388100266456604, -1.7341361045837402, 0.5803136825561523, 0.9162850379943848, -1.3040566444396973, 0.06151023507118225, 0.0633680447936058, 0.853346586227417, -0.31902316212654114, 1.40572988986969, -2.0505244731903076, 0.3847501277923584, 0.5796933174133301, 0.48356738686561584, -0.6577638983726501, -0.16602922976016998, -0.4573281407356262, 0.927585780620575, 0.3785475194454193, -0.16724702715873718, 2.0753681659698486, 0.6959750652313232, -0.6989257335662842, 0.011382230557501316, 0.7197750210762024, 0.6634013056755066, 0.556842029094696, -1.814197063446045, -0.4872300326824188, 1.0300949811935425, -0.8415123820304871, -2.453415870666504, 0.8622246384620667, 0.38864263892173767, -1.6170940399169922, -0.41090235114097595, -0.6512951254844666, -0.3550610840320587, 0.7937425971031189, 0.5250845551490784, -0.3350234925746918, -0.5635189414024353, -0.41546162962913513, -0.8367822170257568, 0.6233473420143127, -0.48403143882751465, -0.004231635015457869, -0.22605782747268677, 0.5821400880813599, 0.9570465087890625, 0.39506107568740845, 0.16535550355911255, -0.839439332485199, -0.7429497838020325, 0.7508479356765747, -0.8559342622756958, -0.7640105485916138, 0.7457790374755859, -1.19669771194458, -1.8961727619171143, 0.2239668369293213, -0.0005004738923162222, -1.266355276107788, -0.7500851154327393, -2.8105814456939697, 0.9168432950973511, -0.9745737314224243, 0.8587728142738342, -0.6643821597099304, -1.0472034215927124, -1.2195945978164673, 0.596690833568573, -0.7458637952804565, -1.3254151344299316, -0.9916723966598511, -0.7120351195335388, -0.33316779136657715, -0.9419716596603394, 0.39175742864608765, 0.6687561869621277, -0.1285734623670578, 0.7985056042671204, -0.6610660552978516, 0.5055391788482666, -0.2555548846721649, 0.3688299357891083, -1.566005825996399, 1.7323365211486816, 0.3883700370788574, 0.42378342151641846, -0.35559529066085815, -0.9825143218040466, -0.3118114471435547, -0.33019018173217773, -1.0098618268966675, 0.30568990111351013, -1.6309800148010254, 0.07263444364070892, -0.1369200050830841, 0.3741724491119385, -0.2309446930885315, -2.572338104248047, -2.2207045555114746, 0.6236501336097717, -1.8411376476287842, -0.6049244403839111, -1.000108003616333, 1.0297365188598633, -1.1846085786819458, -0.8431225419044495, 0.735647439956665, -0.391906201839447, -0.07371331006288528, -0.16452868282794952, 0.16589266061782837, -0.6356736421585083, -0.5841620564460754, 0.7948753237724304, -0.5818417072296143, -0.05333530902862549, -1.0223522186279297, -0.5280243158340454, -1.551517367362976, 1.7817479372024536, -0.3839736878871918, 0.16185887157917023, 0.20414309203624725, 0.5792316794395447, 1.1478925943374634, -2.027470827102661, -0.4196840226650238, -0.20687267184257507, 0.02845938317477703, -1.6993550062179565, 1.921758770942688, -0.1390136182308197, 0.7981510758399963, -0.21880178153514862, -1.194854497909546, -0.7797091007232666, 2.3704960346221924, -0.3226659297943115, 1.7337758541107178, 0.760627269744873, 0.9961827397346497, -0.5090796947479248, -0.15395480394363403, 0.3083283305168152, 0.04377490654587746, 0.27533847093582153, 2.3504698276519775, 0.5290641784667969, 1.023841381072998, -0.38831931352615356, -0.30583223700523376, 0.3096920847892761, 0.8573146462440491, -0.3573797643184662, -0.24221371114253998, 0.5203655362129211, 1.7854292392730713, 0.2961053252220154, -0.8839863538742065, 0.01602671481668949, -0.6859092116355896, 0.1022624745965004, 0.6266534328460693, -0.7870619893074036, -1.6004114151000977, 0.6572718024253845, -0.2853975296020508, -1.4339481592178345, 0.8627114295959473, 0.8231728672981262, 0.5268864631652832, -0.27420222759246826, 0.6130027174949646, 1.8399041891098022, -0.21517275273799896, -0.6404618620872498, -0.33027133345603943, 0.6554560661315918, 0.8100915551185608, 1.9307100772857666, 0.7849507927894592, 0.004621187224984169, -1.3807663917541504, 0.6844805479049683, 0.8242645859718323, -0.536127507686615, -1.5932692289352417, -0.32890555262565613, -0.7507272958755493, -0.8676385283470154, 0.9810526371002197, -0.6246469020843506, 0.6588916778564453, 1.3143376111984253, 0.3709613084793091, -0.7715678215026855, -0.9585708379745483, -0.7252885699272156, 0.6460124850273132, 0.78473299741745, 0.8909911513328552, 0.3486517369747162, 0.2787306606769562, -1.0491602420806885, -0.5158515572547913, 1.3987082242965698, 1.3659212589263916, -0.9965173006057739, -0.8342100977897644, -1.313053846359253, -2.6774260997772217, 0.19798050820827484, -0.6729226112365723, -1.7423079013824463, 1.8273710012435913, 0.3027215600013733, 2.0348150730133057, 1.076522946357727, -1.1307884454727173, -0.09158812463283539, 0.3099445402622223, 0.8937991261482239, -2.733078956604004, -1.487716794013977, 1.0787311792373657, 1.1179699897766113, -0.7386180758476257, -0.452538400888443, -1.5808385610580444, 0.6571369767189026, 1.3711215257644653, -0.16601058840751648, 2.0689258575439453, -0.7680456638336182, -0.05568269267678261, -1.160169005393982, -0.41023731231689453, 0.13254590332508087, -0.6748824119567871, -2.1953494548797607, 0.8921977877616882, -0.5717438459396362, 0.09544447064399719, -0.625848114490509, 0.990854799747467, 0.7756345868110657, -1.3760089874267578, 1.7701613903045654, -1.833775520324707, -0.9263700842857361, 0.2565302848815918, 0.8330951929092407, -1.922237753868103, -0.2681053578853607, 0.06871560961008072, 2.74880313873291, -1.4256175756454468, -1.698310375213623, 1.8656158447265625, 0.09463220089673996, -0.6593962907791138, -0.10367914289236069, -0.6121274828910828, 0.040376439690589905, 0.16085588932037354, 0.22367499768733978, 0.27629753947257996, 0.10034523159265518, 2.039564371109009, 1.3828212022781372, 0.6266534328460693, -0.050868332386016846, 2.6295859813690186, 0.5051757097244263, -0.42214998602867126, -0.3569076657295227, 1.9754557609558105, -1.2296956777572632, -1.4178904294967651, -0.12069089710712433, 2.303389549255371, -0.7592437863349915, 1.0725957155227661, 0.5639238357543945, 0.8845376372337341, -1.1271631717681885, 0.785302996635437, 0.7009288668632507, -0.8500837087631226, -0.8573635816574097, -0.19104361534118652, 1.1569058895111084, -0.7847390174865723, -0.8484026789665222, -1.2594490051269531, -1.8016945123672485, -1.2129583358764648, 0.8104462623596191, -0.4310317039489746, 1.420849323272705, 0.8080422282218933, 0.4581705927848816, 0.9237253665924072, 2.162353754043579, -0.043921515345573425, -0.21147944033145905, -1.2944954633712769, -1.4363751411437988, 0.5675846338272095, -0.5604142546653748, 2.1996009349823, -0.04355696216225624, 0.7034994959831238, -0.20235764980316162, 1.4775183200836182, -0.24390706419944763, -0.10214326530694962, -0.1271241009235382, -1.427567481994629, 0.721983015537262, -0.21265538036823273, 0.08895384520292282, -0.5137495994567871, 1.1394840478897095, 0.09345617145299911, 1.017673373222351, -1.7364996671676636, -0.790665864944458, 0.5845624208450317, 0.22919268906116486, 0.22797547280788422, -1.0374594926834106, 0.2396443635225296, -1.4826701879501343, 0.3378058075904846, -1.9747236967086792, -0.3306849002838135, -0.20892055332660675, 0.36079341173171997, -0.6618839502334595, -0.3162144124507904, 0.7985498905181885, -0.18637345731258392, 1.3981482982635498, -0.16274479031562805, -2.707277536392212, -0.05974749103188515, 0.2257787585258484, -1.4425610303878784, 0.8416748046875, 0.35439321398735046, -1.2149239778518677, 1.491136074066162, -0.4707244038581848, 0.37601345777511597, 0.9552029371261597, -0.08216948062181473, -0.7096303105354309, 1.225571632385254, -0.3269241750240326, 0.21499158442020416, -0.6138585805892944, 1.5714757442474365, 1.0313714742660522, -0.5319862961769104, 1.3606774806976318, 0.08443332463502884, 0.43551725149154663, -1.0227621793746948, -1.3644447326660156, -0.9193152189254761, 0.6936346292495728, -0.48289793729782104, 0.4672255218029022, 0.8074588775634766, 0.04925685375928879, 1.3250303268432617, 0.030695144087076187, -0.5477057695388794, -0.024693256244063377, -0.3994767367839813, 0.5632012486457825, -0.9682509899139404, -0.6028364300727844, -0.9358236193656921, -0.09407226741313934, -0.9531931281089783, 0.9362141489982605, 1.2582271099090576, -0.8796163201332092, 0.13363578915596008, 0.12489967048168182, -0.5300386548042297, -0.1918305903673172, 0.41954201459884644, 0.687962532043457, -0.342957079410553, 1.3598158359527588, -0.8051785826683044, 0.3886305093765259, 1.2025240659713745, -1.4329785108566284, -0.042888887226581573, 0.49216127395629883, 0.5013303160667419, 0.018359744921326637, -1.1432483196258545, 0.08535079658031464, -0.7445358633995056, 0.16916058957576752, -0.4342544376850128, -1.8951351642608643, 1.6697527170181274, -0.9550422430038452, -1.273801565170288, 0.10537413507699966, 1.4021406173706055, -1.375392198562622, 1.0908868312835693, 0.9828338623046875, -0.704108715057373, 1.0502480268478394, -1.317872405052185, -2.2698988914489746, -1.5407487154006958, 0.06164316460490227, -0.2106427550315857, -2.0468122959136963, 0.02908233553171158, 0.5533620715141296, -0.5972262024879456, -2.1274936199188232, -1.4181832075119019, -0.8033208250999451, -0.2365887314081192, -0.7255879044532776, -0.07251280546188354], [1.7512588500976562, -0.7457113862037659, 1.1433236598968506, -0.388871967792511, 1.7394492626190186, -1.5783233642578125, 0.4497741460800171, 0.3338318169116974, -0.646416962146759, 0.60076504945755, -0.5046920776367188, -0.22920358180999756, 0.028828999027609825, -0.40208229422569275, -0.5948247909545898, -0.15993134677410126, -0.28257355093955994, -2.3220038414001465, 0.41468346118927, 0.6944357752799988, -0.2593406140804291, -0.5596873760223389, -2.3944778442382812, -1.5111277103424072, 0.9205989837646484, -0.10986722260713577, 0.0010420293547213078, 0.2574276626110077, -2.367475986480713, 1.221375823020935, 0.15187032520771027, 0.17535844445228577, 0.9410299062728882, -0.8791736960411072, 0.49178043007850647, 0.20115935802459717, 0.35653629899024963, 0.7667186260223389, 0.4825862944126129, -0.740976870059967, -0.13267937302589417, -2.0154199600219727, -1.6491191387176514, -0.1643235832452774, -0.17027151584625244, -1.4948924779891968, 0.17645466327667236, 0.062415365129709244, 0.2900848984718323, 2.539527654647827, -1.2187641859054565, 0.787756621837616, 0.4922751784324646, 0.3534744381904602, 0.36434608697891235, 1.3684067726135254, -0.4347447454929352, 0.16162535548210144, -0.44519221782684326, -0.9958235621452332, -0.6689555048942566, 0.22746749222278595, -1.7340114116668701, 1.4866352081298828, 0.23702113330364227, 0.6181002855300903, -1.110594391822815, -0.05755970999598503, -1.7526768445968628, -0.2701222598552704, 1.189760684967041, 0.05509550869464874, 0.14963100850582123, -1.702680230140686, -0.15437254309654236, 0.23827330768108368, -2.246239423751831, 0.035041630268096924, 1.1383039951324463, -0.6283067464828491, 1.5588687658309937, -0.4368422031402588, 1.10007905960083, -1.232569694519043, 0.20665869116783142, 0.9362244009971619, 0.4844103753566742, -2.1001436710357666, 1.212734580039978, -0.6672598123550415, -0.015780970454216003, 1.9315372705459595, 0.28951096534729004, 0.725517988204956, 0.029533270746469498, -0.6714980006217957, 0.78484046459198, 0.3175424039363861, -1.8872281312942505, -1.035323977470398, 2.8066952228546143, 1.3938833475112915, 1.0038723945617676, 0.11198356747627258, -2.9703176021575928, -0.16667020320892334, 0.9152051210403442, 0.2629145085811615, 0.36209848523139954, -0.9958113431930542, 0.6475620865821838, 0.20164750516414642, 0.579328179359436, 1.2582266330718994, 0.7877733111381531, 0.27604761719703674, -0.46736836433410645, -0.11333011090755463, -0.743486225605011, 0.39976462721824646, -0.746220052242279, 2.536513566970825, -0.3053414523601532, -0.13882622122764587, -1.0842825174331665, 0.40476295351982117, -1.5555438995361328, 0.3631380498409271, 0.06512245535850525, 1.6767113208770752, -0.6476510763168335, -0.813486635684967, -0.9029518961906433, 0.2601090967655182, 1.0230324268341064, 0.44777166843414307, -0.8312220573425293, 0.9401641488075256, -0.4930528998374939, 0.7029739618301392, 0.6072087287902832, 0.4767751693725586, 1.0994187593460083, 0.29518887400627136, 1.2515209913253784, 0.8234880566596985, 0.6639482975006104, -0.016139429062604904, 0.4851175546646118, 0.9355547428131104, 0.7981579899787903, -0.9246551990509033, 0.11604335159063339, 0.8225727677345276, 0.019647587090730667, 1.0150238275527954, -1.9121787548065186, 1.7891665697097778, -1.168229341506958, -1.6007091999053955, -0.45195791125297546, -0.2428986132144928, 1.2487179040908813, -2.0068209171295166, -0.5735042095184326, -1.5203025341033936, 0.4434567093849182, 1.3055604696273804, 0.4402666985988617, -0.5864760875701904, 0.7074339985847473, 0.878322958946228, -0.09959489852190018, -0.33600571751594543, -0.0397331528365612, -0.7271336317062378, 1.2304877042770386, -0.8093706369400024, -0.3790262043476105, -0.060549624264240265, -1.0459684133529663, -0.9781216382980347, -0.7262041568756104, -0.2582657039165497, 0.26842746138572693, -0.7829572558403015, -1.746496558189392, -0.5288652777671814, 1.3843278884887695, 0.8878176212310791, -0.17497451603412628, -0.6953464150428772, -0.7139671444892883, -1.345624327659607, 2.004490613937378, 0.3587785065174103, 0.9134984016418457, -0.5430630445480347, 0.8016635775566101, 0.011945470236241817, -0.12964041531085968, -1.1412104368209839, 0.7177751660346985, 0.31922897696495056, 1.2279102802276611, 0.7129205465316772, 0.4764907658100128, 1.0596336126327515, -1.6161681413650513, 0.43052729964256287, -1.1775481700897217, 1.1690266132354736, 0.9241076111793518, 1.9494248628616333, 1.4387561082839966, 1.9579588174819946, 1.4336439371109009, -0.14936581254005432, 0.47694921493530273, 1.2925609350204468, 0.0593608058989048, -0.002537020016461611, -0.8594454526901245, 0.04419911280274391, 0.9269808530807495, 1.196398138999939, 0.29110491275787354, -1.7614061832427979, -0.7407867908477783, 2.3764030933380127, 0.29644766449928284, 1.105505108833313, -0.4232364594936371, -0.26187270879745483, 1.5098627805709839, 0.20440642535686493, -0.38075730204582214, 1.638290286064148, -1.1265814304351807, -0.5903654098510742, 0.6570552587509155, 0.626799464225769, 1.20333993434906, -0.5565434098243713, 1.1282594203948975, -0.8596152067184448, 0.3497691750526428, 0.6193010210990906, -0.22863365709781647, -0.6204366683959961, 0.225917249917984, -0.11380939930677414, -0.30174729228019714, 0.5084546208381653, 1.4630050659179688, -1.5605696439743042, 1.3267138004302979, 2.083514928817749, -0.08601511269807816, 1.2309166193008423, 0.43396905064582825, -0.5713235139846802, 0.9985524415969849, -0.908704936504364, 1.473484992980957, 0.4233015477657318, -1.973887324333191, 0.4483080804347992, 1.443677306175232, -0.7144785523414612, 1.2758169174194336, 0.6465940475463867, -0.4125840365886688, 0.009805765934288502, 1.5480281114578247, 0.4650394916534424, -1.127100944519043, -1.220956802368164, 0.8014744520187378, 0.8509939312934875, -0.37778744101524353, 0.19388942420482635, 1.1847529411315918, 0.41965046525001526, 1.1022249460220337, -1.2426682710647583, -0.25070062279701233, -1.2450424432754517, -1.3220219612121582, 0.5828787684440613, -0.9865730404853821, -1.7002983093261719, -0.348923921585083, -0.2123904824256897, 0.6695677638053894, -0.32986101508140564, 1.2063989639282227, 0.3273928165435791, 1.2710942029953003, 1.4008164405822754, 0.1549559086561203, -1.5936050415039062, 2.099553346633911, 0.15661674737930298, 1.109627604484558, -0.4387125074863434, -0.3580109477043152, 0.33835235238075256, -1.2152214050292969, 3.262814998626709, 1.477667212486267, -1.9008299112319946, -0.35313236713409424, -0.24775327742099762, 0.8552732467651367, -0.1239241510629654, -0.21445897221565247, -0.02305235154926777, 1.2900010347366333, 0.3687364459037781, -1.0113444328308105, 0.8119659423828125, -0.23520703613758087, 0.624589204788208, -1.1760832071304321, 0.14914675056934357, 0.8498882055282593, -0.12001039832830429, -0.6031474471092224, -1.6614580154418945, 0.104466013610363, 0.4078354239463806, 1.6718132495880127, -1.2339649200439453, -2.5488061904907227, -0.7063152194023132, -0.46207094192504883, -2.3292386531829834, -1.0451642274856567, -1.0138883590698242, -1.6257354021072388, -0.2248414158821106, -0.9087734222412109, -1.3345993757247925, -0.4592859447002411, -1.8289581537246704, -0.47700560092926025, 0.3676929473876953, -0.32516470551490784, 0.41731444001197815, 1.3640273809432983, 1.2669793367385864, -0.21237505972385406, -0.5360814929008484, 0.19737324118614197, 0.31202203035354614, 1.3194483518600464, 1.405625343322754, -0.2290320247411728, -0.2751609683036804, -0.43439894914627075, 0.048569343984127045, -0.2008693814277649, -0.43991366028785706, -0.044568415731191635, -0.9481480717658997, -0.6830251812934875, 0.6123375296592712, -1.0366414785385132, 0.8463771939277649, -0.5162950754165649, 0.8106174468994141, -0.2701708972454071, -0.7603645920753479, 0.1465473473072052, 0.37048768997192383, -1.739355444908142, 0.08983184397220612, -0.1683291792869568, 1.2882096767425537, 0.01939498260617256, 1.2245943546295166, -0.034976329654455185, 1.532413363456726, -0.653722882270813, -0.048542436212301254, -1.6536887884140015, 2.433267116546631, -1.020033359527588, 1.3144179582595825, -0.47331857681274414, -1.3220843076705933, -0.996509850025177, -0.5555887222290039, -0.887213945388794, 1.3010262250900269, 1.481505036354065, -1.78246009349823, -0.0819137766957283, -0.8182470798492432, -0.934307873249054, -0.5467897057533264, 0.2726176977157593, -0.9084805846214294, -0.1308235228061676, 0.6080905199050903, 0.10987982898950577, 0.2466445118188858, -0.17486199736595154, 0.16217559576034546, -0.27593424916267395, 1.1106340885162354, -1.2156649827957153, 1.5138870477676392, -0.9838010668754578, -0.3734007775783539, 1.3192251920700073, -0.458911269903183, 0.5548927187919617, 0.3009382486343384, 1.810562014579773, 0.8046790957450867, 0.7212305665016174, 0.6749887466430664, -0.6360920667648315, -0.6911101341247559, -1.18313729763031, 0.9608153104782104, -0.6234928369522095, 1.6350009441375732, -1.6819649934768677, 1.4598740339279175, 1.1991305351257324, 1.080126166343689, -0.5364922881126404, -2.1092050075531006, -0.45228874683380127, -0.00958366971462965, 0.5429962277412415, -0.07090036571025848, -0.19701877236366272, 1.1879981756210327, -0.31605297327041626, -0.33189141750335693, 1.7163833379745483, 1.246323823928833, 0.7041542530059814, 1.1307004690170288, 2.4078264236450195, 0.5639771819114685, 0.7657549381256104, 0.8742119669914246, -0.5740947127342224, -0.5118879079818726, -0.06424862146377563, -0.06919148564338684, 0.6274206042289734, 1.3811646699905396, 0.055158067494630814, -0.17202849686145782, -0.38578349351882935, -0.44788071513175964, -0.6671035289764404, -0.7274601459503174, 1.4192256927490234, 0.38168516755104065, 0.5209915637969971, 0.8105463981628418, 0.4423205256462097, 1.1720081567764282, 0.7387966513633728, 2.0999867916107178, 1.2442469596862793, -0.34904995560646057, -0.2923054099082947, 2.339463233947754, -0.3291221261024475, 1.2044504880905151, -1.03733491897583, -1.827420949935913, -0.7610834240913391, 0.06988400965929031, -0.03046518564224243, 0.19357673823833466, -0.9566906094551086, -0.4729517698287964, 1.5005098581314087, 0.41008520126342773, 0.5291541218757629, 0.18625785410404205, 1.8048983812332153, 0.5834594964981079, -0.55763179063797, 1.1244913339614868, 0.6917575597763062, 0.3210753798484802, -0.059085696935653687, 0.29200512170791626, -0.6501631736755371, 0.4957679212093353, 1.1390756368637085, 0.7770002484321594, 1.1615748405456543, 0.9276000261306763, -2.1754848957061768, 0.8145732283592224, 0.18445739150047302, 0.2882097363471985, 1.5856053829193115, 0.5940920114517212, 1.0595003366470337, 0.8047727346420288, -0.42848390340805054, -0.19712528586387634, -0.06988265365362167, 0.11297143995761871, -0.19146516919136047, 0.32809752225875854, 1.7158844470977783, 1.234014630317688, -0.6365408301353455, -0.8398747444152832, -0.46906808018684387, -1.0923445224761963, 1.4973268508911133, -0.16705065965652466, -1.4610761404037476, -0.0702277272939682, -0.929832398891449, 0.0047499146312475204, 0.22006800770759583, -0.891430675983429, 0.4268379807472229, -0.10060666501522064, -0.17247280478477478, 1.7965872287750244, -1.831939935684204, 0.25866344571113586, 0.29605957865715027, 1.4810575246810913, 0.5604764223098755, 0.38014665246009827, 0.21895557641983032, -0.7291385531425476, 0.4806770384311676, -0.8323642611503601, 0.5710705518722534, 0.735945999622345, -1.4475748538970947, 0.37006619572639465, -0.19337491691112518, -0.20361021161079407, 1.3216490745544434, 1.7005248069763184, 0.5723637938499451, -1.8264062404632568, -0.7821017503738403, 1.1087915897369385, 1.4851950407028198, -0.04701073467731476, -0.6051833629608154, 1.4298133850097656, 0.5086964964866638, -0.8815529942512512, 0.8035857081413269, -1.0422883033752441, 0.9078981876373291, -0.14248313009738922, 0.19738689064979553, 1.6642584800720215, 0.6415589451789856, -0.16303293406963348, 0.23618817329406738, -1.8381038904190063, -1.064933180809021, -1.3904155492782593, -0.6523855328559875, 0.8857519030570984, 2.821418523788452, 0.45151564478874207, -0.48918360471725464, 0.3568055033683777, 1.111680030822754, -1.4330847263336182, -1.0164010524749756, 0.3257022202014923, -0.9924525618553162, 0.395263135433197, -0.4832642078399658, 1.242994785308838, 0.5237773060798645, -0.8406577110290527, 0.08766654133796692, -0.05285592004656792, -0.049238234758377075, 1.7399535179138184, 0.2659621238708496, -0.26951780915260315, -0.319670706987381, 0.9137647151947021, 1.1645313501358032, -1.860939621925354, 1.5862873792648315, 1.5481458902359009, 0.4242638349533081, 0.7487125992774963, 0.13104508817195892, -2.103090286254883, 1.5838907957077026, 1.0183910131454468, -0.1497485190629959, 0.29811742901802063, -0.8387430906295776, -0.055798985064029694, -0.29053863883018494, 0.42980366945266724, -0.5899831652641296, 1.5699247121810913, 0.014266295358538628, -0.11934787780046463, -0.09368781745433807, -0.022462232038378716, -1.83122980594635, 0.006894982419908047, 2.4712698459625244, -0.3393763303756714, 1.3143976926803589, 2.206209659576416, 0.37823691964149475, 1.1081846952438354, 0.1413150280714035, 0.6709638833999634, -1.214949131011963, 0.4346151053905487, 2.028878688812256, -0.5213384628295898, -1.5757277011871338, -1.8093940019607544, 1.0573621988296509, 0.32784199714660645, -0.06510096043348312, 0.47822779417037964, 0.5146718621253967, -1.0674725770950317, -0.5810272693634033, -1.3701549768447876, -0.9273324012756348, 1.6657452583312988, 1.0148272514343262, -0.4952860474586487, -1.9417686462402344, -0.4727671444416046, 1.2601423263549805, 1.065564513206482, 0.11581622809171677, -1.1549116373062134, -0.01499563455581665, 0.2534911334514618, -1.4032502174377441, -1.6492831707000732, 0.19457878172397614, 0.27222269773483276, -1.308402180671692, -0.5039977431297302, -0.32805994153022766, 1.2678066492080688, -0.29513847827911377, -1.1656429767608643, -1.0351372957229614, 1.2483859062194824, -0.3645210266113281, -1.0621780157089233, -0.9457257986068726, -0.46111443638801575, 0.26444774866104126, -1.0357692241668701, -0.9420556426048279, -1.5314676761627197, -0.9436765313148499, -0.35105106234550476, -0.3571716248989105, -1.0012471675872803, 1.180985689163208, -0.4159002900123596, 2.8508355617523193, -0.4857313632965088, -0.2839233875274658, -0.05657600611448288, -1.2518831491470337, -0.23142988979816437, 0.04319976270198822, 0.9448009133338928, 0.7485160231590271, -0.48356279730796814, 0.6397970914840698, -0.7150816917419434, 1.9433751106262207, 0.27442866563796997, -1.1331782341003418, 1.828331708908081, -0.25068220496177673, -0.3117484748363495, -0.91831374168396, -0.13963255286216736, 0.5652381181716919, -0.5009543895721436, -0.2759896218776703, 0.0337449312210083, -0.8394319415092468, 1.4019861221313477, -0.49650442600250244, -0.13033229112625122, -0.490119606256485, 0.02700124680995941, -1.0083869695663452, 1.1589469909667969, 0.31527820229530334, -0.08295978605747223, -0.22938460111618042, 1.5879851579666138, 1.218296766281128, -0.31701990962028503, -0.038277823477983475, -0.45498859882354736, 1.4267330169677734, 0.0888269692659378, -0.6830881237983704, -2.097637176513672, 1.1622573137283325, 1.0288848876953125, -1.4380929470062256, -0.9407150745391846, -1.062307357788086, 0.6101701259613037, 2.05888032913208, -1.5638090372085571, -1.1243020296096802, -0.9790070652961731, -0.24072280526161194, -1.4210175275802612, -1.577029824256897, -0.7888906598091125, -0.4339279532432556, 1.399182915687561, 1.2077252864837646, 0.5277740359306335, 1.0050792694091797, 2.9993510246276855, -0.5736525654792786, -0.630181074142456, -0.9437959790229797, -1.5724395513534546, -0.8483538031578064, 0.39458414912223816, -0.664372980594635, -0.058203428983688354, -0.6848955750465393, 0.960686445236206, 0.32200899720191956, -1.6472804546356201, -1.0754622220993042, -1.1322989463806152, 0.37688541412353516, -0.1970026195049286, 0.714065670967102, 0.189324289560318, -0.24092736840248108, -0.3224697411060333, 0.8047801852226257, 1.1372787952423096, -0.06513836979866028, 0.5432131290435791, -0.5287436842918396, -0.9152964949607849, 1.5727434158325195, 0.6285969614982605, 0.24513868987560272, 1.244625449180603, 0.20492762327194214, 0.21174119412899017, -0.2099558562040329, -1.8658077716827393, 1.112471342086792, 1.5089964866638184, -0.757215678691864, 1.279793620109558, 0.03720629960298538, 0.5668967366218567, 0.8609183430671692, -0.28125470876693726, -1.0375356674194336, 1.2417773008346558, 1.1217131614685059, 0.7175207734107971, -2.8882782459259033, -0.1579301506280899, -0.5700217485427856, 1.0830782651901245, -0.8885337114334106, 2.2284815311431885, -0.34171268343925476, -0.3510933518409729, -0.7142163515090942, -0.33021342754364014, 0.9042761325836182, -0.6595136523246765, -0.48312216997146606, -0.24728746712207794, -0.4613931477069855, 0.3108876347541809, -0.3511832654476166, 0.9085873961448669, 0.9554965496063232, 0.6666674613952637, 0.6600884795188904, 1.495479702949524, 1.2367289066314697, -1.520520567893982, -1.24013352394104, 1.0266764163970947, 0.35519343614578247, 0.10056543350219727, 0.8470785617828369, -0.11325531452894211, 0.710112988948822, 2.0890326499938965, -0.7536064982414246, -0.30734723806381226, -0.8811129927635193, -1.3761203289031982, 0.24516648054122925, -0.8965286612510681, 0.8180366158485413, -0.23243120312690735, -0.19199834764003754, -1.2775520086288452, -0.3597581684589386, -1.3533580303192139, 0.10767567157745361, -0.8323834538459778, 0.9752166867256165, -0.3426235020160675, -1.1386741399765015, 0.9728137254714966, -1.315924048423767, -0.5180917978286743, -1.822824239730835, 1.0807605981826782, -0.316393107175827, -2.210662364959717, 0.7341383695602417, 0.16962696611881256, -0.6548655033111572, -0.2179461270570755, 1.0991065502166748, 0.17136231064796448, 0.7869964241981506, -1.7506746053695679, -1.367101788520813, 0.9768843054771423, 0.7325286865234375, -1.0139150619506836, -0.006659830454736948, -0.26425009965896606, 1.9645296335220337, 1.112542748451233, 0.6330974102020264, 0.334036648273468, -1.756919503211975, 1.0386297702789307, -0.5110220909118652, -0.423216849565506, 2.306464195251465, 0.27745503187179565, 0.8296013474464417, 0.7087792754173279, 1.360594630241394, -1.14188814163208, 1.178444266319275, -0.06369383633136749, -0.22321994602680206, 0.16662020981311798, -0.5487003922462463, -1.1533637046813965, 2.0269978046417236, 2.1262099742889404, -0.7570133805274963, 0.18267816305160522, -0.08205421268939972, 0.41319072246551514, 0.4925796687602997, 0.563305139541626, -0.22388698160648346, 0.4208628237247467, -0.121538907289505, -1.2288974523544312, 1.039055585861206, 0.745132327079773, 1.1370309591293335, -1.0683659315109253, 0.3838142156600952, -1.3700319528579712, -0.5492367148399353, 1.0144331455230713, -0.8393399715423584, 1.233573079109192, -0.3539379835128784, -1.2660077810287476, -1.3836122751235962, 0.3894106447696686, -0.6360855102539062, 0.39194419980049133, 1.0566966533660889, -1.1833381652832031, 0.7632355093955994, 0.825628399848938, 1.1956439018249512, -0.30893582105636597, 0.1409432590007782, 2.2633237838745117, -0.20991849899291992, -0.5835155248641968, -1.1183490753173828, 0.18324382603168488, 1.039599061012268, -0.45761391520500183, 1.3690840005874634, 0.6092401146888733, -0.13921545445919037, 0.9956107139587402, 1.1788055896759033, 0.503596305847168, 2.016216516494751, -0.7366614937782288, -2.134349822998047, -0.7341724634170532, -1.1608378887176514, 1.19260835647583, -0.8589369654655457, -1.181761384010315, 0.8111398220062256, 0.1451427936553955, 0.38063108921051025, -1.1009362936019897, 0.023251939564943314, -0.14878301322460175, 2.7178850173950195, -0.9011985063552856, -0.8754658102989197, 0.5795486569404602, -0.08466639369726181, -1.0813429355621338, 0.2720392048358917, 0.5355812311172485, -0.33248084783554077, 0.9514827728271484, 0.5989218354225159, 0.9783052206039429, -0.10988061875104904, 0.3653198778629303, -0.4265783131122589, 0.41361677646636963, -1.593514084815979, -2.434487819671631, 0.5782538056373596, 0.032889142632484436, 0.9727543592453003, 0.23267675936222076, -0.24033907055854797, 0.938980221748352, 0.6084961891174316, -0.22601033747196198, -2.137985944747925, -1.578830599784851, 0.5968910455703735, -0.29680296778678894, 1.0592987537384033, -0.1820855289697647, 0.15305419266223907, 0.2079344540834427, 0.12995146214962006, 0.4867497980594635, -0.3187717795372009, 2.253117561340332, -0.2774919867515564, -0.028574414551258087, 0.0029230965301394463, -0.07120049744844437, -1.3599778413772583, -1.3294756412506104, -1.2338758707046509, 0.8529098033905029, 0.2435641586780548, -0.28812727332115173, -0.9357994794845581, -0.6422608494758606, -0.8632980585098267, 1.4356906414031982, 0.3828694522380829, -1.3261919021606445, 0.9886267781257629, 0.8124220371246338, -1.571977972984314, 0.09233113378286362, -0.5351399779319763, -1.8870879411697388, -1.7460975646972656, -2.434335708618164, -0.019511550664901733, -0.3917754292488098, 1.3360694646835327, 1.077457070350647, -1.4514201879501343, -0.9886162281036377, 1.0509573221206665, -2.16145658493042, 0.004800212103873491, 2.0909483432769775, 0.6357371211051941, 1.5036990642547607, 0.3229973614215851, -1.926658034324646, -0.8322573304176331, -0.04935722053050995], [-0.9924454092979431, -0.2928299009799957, 0.31984618306159973, -1.4088350534439087, -0.17168603837490082, 1.59946870803833, -0.8037264943122864, -0.07380612194538116, -2.160921573638916, -1.993216872215271, 0.0015360810793936253, -0.029461493715643883, -2.523064136505127, 0.5077645778656006, -0.02956882119178772, 0.6953184604644775, 0.47869616746902466, 0.7920768857002258, 0.3088049292564392, -0.1984470784664154, -0.2857458293437958, -0.7127547264099121, -1.6036927700042725, 0.8152679204940796, -0.1768532395362854, -0.6976020932197571, 0.7368157505989075, -0.28826776146888733, 0.29609787464141846, -1.948533058166504, 0.042413223534822464, 0.30298399925231934, -0.005693649407476187, 0.19359827041625977, 0.8633347749710083, -0.8287568092346191, -0.8740217685699463, -0.4014604389667511, 0.4034700095653534, -1.027299404144287, 0.3726056218147278, -2.2630815505981445, 2.2947633266448975, 1.082471489906311, -2.0511322021484375, 0.23919036984443665, -1.0841635465621948, -0.5247429609298706, -1.5921260118484497, -0.7992884516716003, 0.28912875056266785, 0.8493687510490417, 1.5421345233917236, 1.457916498184204, 1.1912293434143066, 1.1706352233886719, 1.0600266456604004, -1.5605320930480957, -0.48500120639801025, 1.7838172912597656, 0.082571841776371, 0.010344387963414192, -2.1543779373168945, -0.6012186408042908, 0.6551218032836914, 0.2620651125907898, 0.7029968500137329, 1.1292587518692017, -0.701004147529602, -0.6308438777923584, -0.7383936643600464, 0.2723667621612549, -1.3967341184616089, 0.7485730648040771, -1.8066959381103516, -0.8101491332054138, 0.4209062159061432, 0.485726535320282, 0.6683940291404724, 0.8090634346008301, -0.2400253862142563, 1.0380765199661255, -0.01805492863059044, -0.4614534080028534, 1.3933745622634888, -0.1765184849500656, 0.7589951753616333, 1.3614182472229004, 0.009721918031573296, -0.5476102828979492, 0.049860451370477676, -0.6530250906944275, -1.0202442407608032, 0.3011421859264374, -0.4220443367958069, -0.9899070262908936, 1.1912740468978882, -1.2761150598526, -0.6254205703735352, 2.7306857109069824, 0.7112343311309814, -1.7232534885406494, 0.32767248153686523, 1.2482500076293945, 1.1201517581939697, -0.6349062323570251, 0.09841159731149673, -0.01171258557587862, -0.07970233261585236, -1.0362783670425415, -0.7311115264892578, -0.7367151379585266, 1.1448289155960083, -0.924613893032074, 0.8172740340232849, 1.0364302396774292, 1.1162333488464355, -0.5264813303947449, -0.6759559512138367, -0.17443177103996277, 0.21305295825004578, 1.0370829105377197, -0.6914886236190796, -1.5962791442871094, 0.019824979826807976, -0.8327258229255676, -0.10779670625925064, 0.6358372569084167, -1.9103275537490845, 0.8059043288230896, -1.8501583337783813, 0.6840893626213074, 0.2952318489551544, 0.38790032267570496, -0.18801060318946838, -1.5157357454299927, -0.9224331974983215, -0.20222464203834534, 2.016512393951416, 1.1055415868759155, 0.15907126665115356, -0.2976490557193756, 0.2447281777858734, -0.6408929228782654, 0.48451390862464905, 0.4777269661426544, 0.8695144057273865, 0.6392156481742859, -0.5185126066207886, -3.6614034175872803, -1.6863319873809814, -1.2926623821258545, -1.070372223854065, 0.185003861784935, -0.730322539806366, -0.5827509164810181, 0.18606239557266235, 1.4133706092834473, 0.8290225267410278, 0.12315581738948822, 0.34986230731010437, -2.226142644882202, 0.24500513076782227, -1.4554730653762817, -0.5302507281303406, -0.4990272521972656, 0.6177359819412231, -2.076364517211914, 0.05084679275751114, -0.45869243144989014, -0.0642385259270668, 0.3206538259983063, 1.6963086128234863, -1.2059099674224854, -0.001585672958754003, -1.0965907573699951, 0.4358270466327667, -1.6116020679473877, 0.9494695067405701, 0.5841812491416931, 0.4474693834781647, 0.18582825362682343, 1.1701949834823608, -2.1603641510009766, -0.08702727407217026, -0.702467143535614, 0.5457606911659241, -0.5798053741455078, -1.5028910636901855, 1.5453482866287231, 0.17066513001918793, -1.6988321542739868, -1.2081314325332642, 1.278568983078003, -0.1727685034275055, 1.1041293144226074, 1.3461694717407227, 1.1925147771835327, 1.2830580472946167, -0.40894389152526855, 1.158699631690979, -0.20291079580783844, -0.35069864988327026, 1.0112545490264893, 1.3807859420776367, -0.45320647954940796, -0.11237134784460068, 0.609670102596283, -0.2501332461833954, 1.0437471866607666, -0.36941760778427124, 0.3544078469276428, -0.6153152585029602, 0.9027530550956726, -1.3896764516830444, 2.224003314971924, 0.8730586171150208, -2.1051414012908936, 0.6386905312538147, 2.1406869888305664, 1.2598683834075928, -0.13066332042217255, -1.5369508266448975, 0.6255673766136169, 1.1865423917770386, 0.05339586362242699, 0.6692507863044739, -0.532534658908844, -0.11845245957374573, 1.2573816776275635, 0.19320623576641083, 2.8019564151763916, 1.715728521347046, -0.39889127016067505, -0.305372416973114, -0.16847167909145355, -0.33258095383644104, -1.1593440771102905, -0.00012652233999688178, -2.9000978469848633, -0.2719450294971466, 1.0294039249420166, 0.6937053799629211, 1.664843201637268, 0.3240501880645752, -0.2737019658088684, -0.4234027564525604, -1.5918610095977783, 0.2950810194015503, 1.0583382844924927, -0.9949524998664856, -0.8158000111579895, -0.9564571380615234, 0.805399477481842, 1.862229585647583, -1.3344650268554688, -0.2102833390235901, 0.33863261342048645, 0.24475771188735962, -0.9228729009628296, 0.6362850666046143, 0.18561963737010956, -0.05755595490336418, 0.14525587856769562, 1.0403941869735718, 0.42273759841918945, -1.2403254508972168, -0.2826733887195587, 2.5221505165100098, -1.797788381576538, 0.033724382519721985, 0.7517934441566467, 0.460294634103775, 1.4600915908813477, -0.39379024505615234, -0.9319497346878052, -2.495577096939087, 0.852340579032898, 1.976009726524353, -0.2100839763879776, -1.0728225708007812, 0.3413783311843872, -0.45895978808403015, 0.17124293744564056, 0.171579509973526, -0.9838945269584656, 0.5161138772964478, -0.045412685722112656, 0.8519346117973328, -1.4449105262756348, -0.678022563457489, -1.3746417760849, 0.11773446947336197, -0.19732379913330078, -0.46392205357551575, 2.0330810546875, 0.23345805704593658, 0.17004449665546417, 0.08566251397132874, 0.3602636456489563, -0.027801111340522766, -1.7632337808609009, 0.252469927072525, 1.4456130266189575, -0.12806838750839233, 0.26140567660331726, -2.925136089324951, -1.2375906705856323, -0.5497033596038818, -0.12599477171897888, -0.6388742327690125, 1.461328148841858, 0.30653658509254456, 0.07426529377698898, -1.4462785720825195, -1.2917048931121826, -1.3367655277252197, -0.20092929899692535, -1.1419962644577026, 1.1397173404693604, 0.8424720764160156, 1.810170292854309, 0.1796136051416397, 0.4254712164402008, -0.13625739514827728, -1.6012585163116455, 0.6732926964759827, 0.3526400327682495, 0.6242714524269104, 0.6019817590713501, -1.3945344686508179, 0.2908775508403778, 0.4427117109298706, 0.38639384508132935, -0.23151308298110962, 0.999078631401062, 0.6339569091796875, -0.007156490348279476, -1.406096339225769, 0.4462607800960541, 0.002061475533992052, -0.15373939275741577, -0.4670083522796631, -1.262311577796936, -0.34935981035232544, 0.41527411341667175, -1.4740207195281982, 0.5595574378967285, -0.048345740884542465, 0.25665947794914246, -0.22610536217689514, -0.1288197934627533, -0.7179489135742188, 0.30413347482681274, 0.6333046555519104, 0.12163294106721878, 0.944907546043396, 0.019984908401966095, -2.0971503257751465, -1.187730073928833, -1.0658416748046875, -0.5602035522460938, -0.587884247303009, -0.5625385046005249, -0.4693150818347931, 1.1599515676498413, -0.7906050682067871, 0.8844040632247925, -0.5059953331947327, -1.2221752405166626, -1.3851189613342285, -0.3500157594680786, -0.7062864899635315, -2.324070692062378, 0.04793388769030571, -1.1918126344680786, -0.09433099627494812, -0.2182256132364273, -0.22354087233543396, 0.40157753229141235, -1.732569694519043, -1.654796838760376, -0.12958768010139465, -0.7458935379981995, 0.5221929550170898, 0.3593887984752655, -0.4047556221485138, 1.1500407457351685, 0.15565456449985504, -1.816813588142395, -0.7810048460960388, -1.0686402320861816, -0.4143780469894409, -1.6636099815368652, -1.3739643096923828, -0.4789029359817505, -0.3629710376262665, -0.6473026871681213, 0.6793310046195984, -0.3706589639186859, -0.4134095311164856, 1.3901827335357666, -0.35866013169288635, 0.9091075658798218, -0.40930381417274475, -0.30231422185897827, 1.0157831907272339, -0.6456352472305298, -0.2722228169441223, -0.7087311148643494, -0.41271844506263733, 0.07300720363855362, -0.7722781896591187, 0.02308620512485504, -0.07990453392267227, 0.9005299806594849, -0.30186301469802856, -0.055345457047224045, 1.6417843103408813, 0.6203218102455139, 0.5021070241928101, 0.7391548156738281, -2.543191909790039, 0.9108182191848755, -0.9520581960678101, -0.6387400031089783, -0.3936830759048462, -0.9316583275794983, -0.00498999422416091, -0.7939579486846924, -0.21793383359909058, -0.28354230523109436, -0.2009475976228714, -1.373661994934082, -0.932735800743103, -1.1897622346878052, 0.6306987404823303, 0.041929859668016434, -1.5217938423156738, -1.1935594081878662, 0.8132145404815674, -2.025876998901367, 0.35802093148231506, -2.024733543395996, -1.506630539894104, 1.1578140258789062, -0.13374488055706024, -0.6753871440887451, 0.11393716186285019, -0.8676454424858093, -0.23213210701942444, 0.3842175602912903, -0.5135388970375061, 1.7067127227783203, 2.1611850261688232, -1.7552950382232666, -0.08672846853733063, -0.3977389335632324, -0.44193869829177856, 0.5609551072120667, 2.5538761615753174, 0.7583000063896179, -0.6728140711784363, -0.637690007686615, -2.9058210849761963, -1.0808793306350708, 1.0502126216888428, 1.3497593402862549, -0.39928218722343445, 1.028743863105774, 0.8874953389167786, -0.2480766773223877, 1.418465495109558, -0.059295255690813065, 1.3930003643035889, -0.688150942325592, -0.5825297236442566, 1.0992989540100098, 0.1448812484741211, -0.45691031217575073, -0.41642236709594727, 0.40983566641807556, -0.7147144675254822, 0.9128528237342834, -0.7352779507637024, -0.631754994392395, -1.182845950126648, 0.8687679767608643, 0.7558925151824951, 1.143413782119751, -0.19757628440856934, 0.4920324385166168, -0.4990094304084778, 0.38881611824035645, 0.0016392979305237532, 0.06446148455142975, -0.34802311658859253, -2.207317590713501, 1.6876258850097656, 1.1547540426254272, -0.04431894049048424, -0.18178890645503998, -0.16344015300273895, 0.4761316776275635, 0.06426975876092911, -0.46388015151023865, 0.16377557814121246, -1.2443485260009766, -0.503780722618103, -0.3418746292591095, 0.15221023559570312, -0.7196331024169922, 1.8210421800613403, -0.9824369549751282, 0.018617335706949234, -1.03697669506073, 2.6916680335998535, 0.5503435730934143, 0.045367173850536346, -0.151565819978714, 1.0046052932739258, -0.49599015712738037, 0.5875715017318726, -0.042391467839479446, 2.5968475341796875, 0.7054048776626587, 1.18406081199646, 0.08798729628324509, 2.6136608123779297, -1.2732130289077759, -0.8114115595817566, -1.0097790956497192, -1.3294947147369385, 0.005753287114202976, -0.11743253469467163, -0.3300197124481201, -0.9239482283592224, -2.102088689804077, 0.28138113021850586, -1.4018173217773438, 0.992891252040863, -0.466275691986084, -1.9564841985702515, 1.1566879749298096, 0.16334368288516998, 0.125311478972435, 0.5601521134376526, 0.13831084966659546, -1.691684603691101, 1.1480306386947632, 0.8954542875289917, 1.7605870962142944, 0.8002618551254272, 1.6212221384048462, 1.4476306438446045, -1.7829999923706055, -1.2105849981307983, 0.42850667238235474, 1.7970023155212402, -0.4993732273578644, -0.03843699395656586, 0.48341041803359985, 1.313450574874878, -0.271897554397583, 1.3962342739105225, 1.0654547214508057, 0.46140503883361816, 1.1844191551208496, 1.276474952697754, -0.4355242848396301, 0.02949991635978222, -0.1135672926902771, -1.3920811414718628, 1.4561203718185425, 0.8775331377983093, 1.9311765432357788, -2.4442784786224365, -1.0939157009124756, -0.22234119474887848, -1.0342777967453003, 0.6643989682197571, -0.6713203191757202, 0.9488252997398376, -0.2833978831768036, -0.9477036595344543, 1.7572065591812134, -0.2370515912771225, 0.16215817630290985, -0.13326098024845123, -0.5670610666275024, 0.36047840118408203, -0.9432286620140076, 1.1178659200668335, 0.8526769280433655, 1.5183793306350708, -0.03230833262205124, 0.6457264423370361, 1.286784291267395, 0.9008241891860962, 0.5432790517807007, -0.030957354232668877, 0.6707393527030945, 1.6310834884643555, -2.9603004455566406, -1.6070775985717773, 0.5875906348228455, 0.2844439148902893, -0.6619691848754883, 0.24955077469348907, -1.3304861783981323, 0.5878168940544128, -2.0268704891204834, -0.6440477967262268, -0.47033435106277466, -0.6431529521942139, 1.00784170627594, 0.7855373024940491, 0.34609314799308777, 0.9899824261665344, -0.34588998556137085, -0.9464945197105408, -1.4356101751327515, -1.585256576538086, 0.5841642022132874, 1.3467034101486206, 0.11749865114688873, -0.09326232969760895, 0.476725697517395, -0.8967710137367249, -0.11003153026103973, 0.42139798402786255, -1.3826247453689575, 1.1880731582641602, 0.5399918556213379, -0.060465823858976364, -0.4304937422275543, 1.7650165557861328, -0.707925021648407, -1.5838085412979126, 0.7872852087020874, -0.6474681496620178, -1.260190725326538, -0.9842870831489563, 0.6458089351654053, 0.5641712546348572, 0.008117551915347576, -0.7418854236602783, 0.017002983018755913, 0.7402066588401794, -0.8984939455986023, 0.2520580291748047, -0.4012097716331482, 1.3102556467056274, -1.17337965965271, 1.3565906286239624, 0.46231502294540405, 0.8233457207679749, -1.8639734983444214, -0.7320986986160278, 0.47244054079055786, -0.18445922434329987, 1.8091853857040405, -0.6949814558029175, 0.8131946325302124, -0.39667096734046936, -0.21374255418777466, -0.3943335711956024, 1.4311020374298096, -0.7795848250389099, 2.2788050174713135, 0.510229229927063, -0.03519001975655556, 0.2561839818954468, 0.6700476408004761, 0.7722157835960388, -0.6523826122283936, -0.7095800638198853, 1.260902762413025, -0.48646044731140137, -0.9060189723968506, -0.4236766993999481, -1.3437265157699585, 1.112765908241272, -0.0400162898004055, 0.9995666146278381, 0.7936357855796814, -0.0005758382612839341, 0.7742542028427124, 0.3118889331817627, -0.4327118694782257, -0.21138933300971985, 0.025775939226150513, -0.31863391399383545, 0.9371010065078735, 0.7303121089935303, -1.5374557971954346, -0.49150845408439636, 1.398644208908081, 1.1812949180603027, -1.085843801498413, 0.7614068984985352, 2.1954941749572754, 0.06120757758617401, 0.634609043598175, 0.840406596660614, 0.6532037258148193, 0.47303083539009094, -1.925872564315796, -0.4361484944820404, -0.7081842422485352, -0.3954591453075409, -0.6538064479827881, 0.21887628734111786, 1.104542851448059, -0.16833612322807312, -0.052240606397390366, -0.14728982746601105, 0.11692287027835846, -1.2959799766540527, -0.807298481464386, -1.2268414497375488, 0.41509950160980225, -2.076204538345337, 0.8800622224807739, -0.5218619704246521, -0.04004272446036339, -0.6446232199668884, -0.03842019662261009, -0.07720378041267395, -2.4225995540618896, -0.12386815249919891, -1.433529257774353, 0.23777151107788086, 0.24191124737262726, -1.6720809936523438, -0.3183828890323639, 1.010231375694275, 1.4652985334396362, -0.3541752099990845, 0.7375858426094055, 1.5507051944732666, -0.14960837364196777, 0.49768200516700745, -1.0998620986938477, 1.1632214784622192, -0.22611166536808014, -2.6913976669311523, 1.1708197593688965, 0.11362964659929276, 2.0122358798980713, -0.112905353307724, -0.8476520776748657, 2.0963079929351807, 2.034924268722534, 0.15551726520061493, -0.8841354846954346, 0.09350234270095825, 1.0347429513931274, 1.3947737216949463, -1.5748924016952515, -0.5415642261505127, -1.8564552068710327, 0.20049066841602325, -1.0796211957931519, 1.5812143087387085, 0.6677320003509521, -1.0246471166610718, 1.1710139513015747, -2.6345057487487793, -0.7041729092597961, 1.1589852571487427, 0.057840488851070404, -0.49304789304733276, 0.6388450860977173, 1.0843861103057861, 0.9818205237388611, -2.096670150756836, 0.7047737240791321, -0.9805216789245605, 0.7199631333351135, -0.07934136688709259, 0.128348708152771, 1.02409029006958, 0.568158745765686, 0.4828781187534332, 1.5053997039794922, -0.2482665479183197, 1.0784207582473755, 0.31928086280822754, 0.1728336215019226, 0.3958483338356018, 0.976742148399353, -1.7909663915634155, -0.36740386486053467, 0.3602145314216614, -0.8238074779510498, 1.0710322856903076, -0.8965409994125366, 1.6865860223770142, -0.33579814434051514, -2.0547165870666504, 0.05099870637059212, -0.5380922555923462, 1.5794209241867065, 0.8924486041069031, 0.5838225483894348, 0.2266474813222885, 0.42089539766311646, 1.4887222051620483, 0.5552242994308472, 1.1837238073349, 0.35323357582092285, -0.41068965196609497, -0.48443537950515747, -0.2811143696308136, -0.41086748242378235, 1.551910161972046, -0.471630722284317, 0.612190306186676, -0.7166857123374939, -0.9533534049987793, 0.7564529776573181, 0.7071521282196045, 0.0529545396566391, -0.03163831681013107, -1.1005254983901978, -0.20717991888523102, -0.24790039658546448, 0.4437721073627472, 0.43385499715805054, -2.2886643409729004, 1.3148223161697388, 0.8123100996017456, -0.1903190314769745, -0.44069603085517883, -0.5821008682250977, -0.907711923122406, 0.31384095549583435, -0.7740008234977722, 0.8409478068351746, -0.7594277262687683, -0.7875990271568298, 1.9965009689331055, -0.30851316452026367, -1.6864546537399292, -0.8093181252479553, -0.4202285408973694, -1.3116225004196167, -2.046729564666748, 0.40259498357772827, -0.9218406081199646, 0.5107518434524536, 0.4662500321865082, 1.4586864709854126, 1.9784331321716309, -0.10185384005308151, 1.8163622617721558, 0.7062056660652161, 1.0838778018951416, 0.5456988215446472, -1.3784223794937134, -0.9185861945152283, 0.8160918354988098, 0.9482778310775757, -0.04985693097114563, 1.5825793743133545, 0.731911838054657, -0.20475229620933533, -1.4903923273086548, 0.15305101871490479, 0.5649668574333191, -0.1454041749238968, 0.18439339101314545, 0.9645360708236694, -0.49489647150039673, 0.47999638319015503, -0.006756684742867947, 0.4413031339645386, 2.0938496589660645, 2.1017637252807617, -1.3662229776382446, -0.7771000266075134, 1.007452368736267, -0.1580997109413147, 0.7448075413703918, 0.30954939126968384, -0.4558369815349579, -1.104831337928772, 1.598090648651123, -1.6659456491470337, 0.5145938396453857, 1.7896790504455566, -1.2754549980163574, 0.013193348422646523, 1.5015794038772583, 0.8322176337242126, 1.5745409727096558, -2.0379490852355957, 0.17956434190273285, 0.14880450069904327, -0.5097811818122864, -1.4896117448806763, 1.5264848470687866, -0.4550168812274933, -0.2622540593147278, 0.2632211446762085, 1.2939471006393433, 0.5072706341743469, 0.44980737566947937, 0.4812578558921814, 0.27692195773124695, 0.2301912009716034, 1.624868631362915, -0.5283516049385071, 1.6190948486328125, 0.555958092212677, 0.8054121732711792, -1.2214267253875732, -0.9353485107421875, -1.386447787284851, -0.08242097496986389, 0.13676118850708008, 0.6287361979484558, 0.3463023900985718, -0.38592034578323364, -1.6325818300247192, 0.924607515335083, -1.5007798671722412, 0.718281090259552, 2.020655632019043, 1.230162262916565, -1.2922009229660034, -0.8975929021835327, 0.15105631947517395, -0.46276506781578064, 0.28671324253082275, 0.0722697302699089, 1.3292827606201172, -1.3224523067474365, -1.1843197345733643, -0.6746987104415894, -0.7682623863220215, -0.44683608412742615, 0.2990802228450775, -0.6568151116371155, -0.6570649147033691, -0.4093988835811615, -1.3084172010421753, -0.33029624819755554, 2.311075210571289, 0.2343718558549881, 0.6312289237976074, 0.32407206296920776, -0.18982861936092377, -0.13440951704978943, -0.23299407958984375, -1.1930981874465942, 0.3027726709842682, -1.0383331775665283, -0.26256614923477173, -2.7206029891967773, 0.1936999410390854, 1.3746089935302734, -0.052295438945293427, -0.4221695363521576, -0.8619887232780457, 1.6656795740127563, -1.0043561458587646, 1.5402778387069702, 1.8049590587615967, -0.32747969031333923, -0.06440364569425583, 0.687312126159668, -0.16691364347934723, -1.104282021522522, -0.3129265606403351, 0.48252591490745544, -0.36514192819595337, -0.16830648481845856, -0.9692253470420837, -1.4345389604568481, 0.957697868347168, 0.6732211709022522, -0.19805391132831573, 1.694291591644287, -0.8029627203941345, -0.12907880544662476, -0.7532558441162109, 0.9696542024612427, -0.38546082377433777, -0.3354700207710266, -0.7133428454399109, 0.16146594285964966, 0.36445319652557373, -1.933637022972107, 0.5158733129501343, 0.18103529512882233, -0.6323835849761963, -0.07564246654510498, -0.8849470019340515, -0.22354109585285187, -0.09565064311027527, 0.11930976808071136, 0.24516619741916656, -0.13325849175453186, -1.3978192806243896, -1.9945541620254517, -0.9153308868408203, 1.0493618249893188, 0.9721345901489258, 1.4494102001190186, -0.35590553283691406, -0.8808119893074036, -1.0171438455581665, 0.049986474215984344, 1.2120025157928467, -0.44758614897727966, 1.1969764232635498, 0.15986081957817078, 1.9545937776565552, -0.6438222527503967, 1.2088673114776611, -1.0615217685699463, 1.0454374551773071], [0.06933528929948807, -0.2573072016239166, -0.1315913051366806, -1.151942253112793, -0.5198054909706116, 0.5601479411125183, 0.48252907395362854, 0.28815093636512756, -0.4880606532096863, -0.15206895768642426, -0.3478876054286957, 1.6027008295059204, -0.3818141222000122, 0.24216797947883606, 0.1869027316570282, 0.8789945840835571, 0.6405261754989624, 0.6430503129959106, 1.2185323238372803, -0.02308581955730915, 0.20721615850925446, -1.2913322448730469, -0.09098842740058899, 0.6203785538673401, 0.9028403759002686, -0.7504833340644836, -0.9490211009979248, -0.7432156801223755, -1.1443809270858765, -0.6007959842681885, -0.84546959400177, -1.4492158889770508, -0.6780946850776672, 0.6409040093421936, -0.3180014193058014, 2.054213047027588, 0.0639922022819519, -1.6051709651947021, -2.1487655639648438, 0.31759756803512573, 0.9056628346443176, -0.18916672468185425, -1.2427934408187866, -0.26172253489494324, 0.6400488615036011, -2.162625789642334, -0.29210641980171204, -1.198051929473877, -0.18757620453834534, 0.6396574378013611, -0.02746696211397648, -0.5818396806716919, -0.5891067385673523, -0.2749526798725128, -0.624218225479126, -1.87798011302948, -1.107143759727478, 0.6319367289543152, 0.7641941905021667, -0.02517627738416195, 0.04320414736866951, 0.19450990855693817, -1.0817975997924805, -2.071340799331665, -0.14096783101558685, -0.5554232001304626, 0.6063435077667236, -0.4749918580055237, -0.029631292447447777, -0.8544743061065674, 0.503446638584137, 0.2028152495622635, 2.111488103866577, -0.5257149934768677, -1.8587490320205688, 0.12457340210676193, 0.061145443469285965, -0.8422579765319824, 1.3490049839019775, -2.5546886920928955, -0.5756915211677551, 0.5547603368759155, -0.3210352957248688, 1.1452988386154175, 0.5281450748443604, 0.10869520902633667, 0.7389966249465942, -2.172907829284668, 1.0226866006851196, 1.8709509372711182, 1.0292531251907349, 0.5790686011314392, -0.9226065278053284, -0.08854759484529495, 0.7172508835792542, 1.5906169414520264, 1.6715303659439087, -0.9157495498657227, 2.3108131885528564, 0.5572519898414612, 0.3752763867378235, 0.14818209409713745, 1.1823089122772217, 0.6585137844085693, -1.2897588014602661, 0.04408368468284607, -0.09248582273721695, 0.7729787230491638, -0.5129476189613342, -0.6162242293357849, 2.4592251777648926, 0.9722496867179871, -0.8344069123268127, -0.3973273038864136, -2.2411909103393555, -1.314191460609436, -1.075391411781311, 0.023170428350567818, -0.17356877028942108, -0.7698196768760681, -2.066617250442505, 0.22934119403362274, -0.2900699973106384, -0.3024105131626129, 2.137625217437744, -0.49898162484169006, -0.9063442349433899, -1.0360264778137207, 0.9755211472511292, 1.5949981212615967, -0.37965622544288635, 0.28847214579582214, -0.4658545255661011, 0.08917611092329025, 0.027648011222481728, 1.2202333211898804, 0.2868553102016449, -1.7708171606063843, -0.8028854131698608, 1.267216444015503, -0.04040323942899704, 0.7546513080596924, -1.5610755681991577, 0.5607180595397949, 0.08033480495214462, -1.1607434749603271, -0.6207630634307861, 2.060776948928833, 1.3398655652999878, 0.2471768707036972, -0.30589956045150757, 0.3588893711566925, -0.9749391078948975, 0.4540577828884125, 1.4344350099563599, -0.07904139906167984, -0.28601908683776855, -1.8164788484573364, 1.9528467655181885, -0.8385924100875854, 0.1690473109483719, 0.7818837761878967, 0.6735032200813293, -1.4272761344909668, -0.5414518117904663, 0.05139369145035744, -0.45189085602760315, -2.0139458179473877, 0.26823750138282776, 0.5996161699295044, 0.2101386934518814, -0.4745902717113495, 1.4315091371536255, 0.23371994495391846, -2.2718546390533447, -0.24965231120586395, -0.32486438751220703, 0.16770268976688385, -0.3981705904006958, 0.08374956250190735, 0.6368457674980164, 0.6497102379798889, 0.11306748539209366, -0.508832573890686, -1.308373212814331, 0.1250067949295044, -0.7354228496551514, 0.042931511998176575, 0.09857768565416336, 0.20334500074386597, -1.7411720752716064, -1.206653356552124, -0.030533315613865852, -0.0643598660826683, -2.3887946605682373, 0.19734762609004974, -2.4744017124176025, -0.04983709752559662, 0.22477641701698303, -0.11442515254020691, -1.745928406715393, 0.07476720958948135, 1.2912055253982544, 1.6182878017425537, 0.6242632865905762, 1.77694833278656, -0.006411215290427208, -1.7647701501846313, 1.5395597219467163, -0.9270051121711731, -0.1583203226327896, -0.6936795115470886, -1.935788869857788, -0.4098077714443207, 0.38691017031669617, 0.6567575335502625, -0.6855915188789368, 0.6638206243515015, 0.2802784740924835, 0.14259979128837585, -0.38315653800964355, 1.2773858308792114, 1.1690332889556885, -1.9357656240463257, 0.16922102868556976, -0.4129007160663605, 1.0960568189620972, -1.0407819747924805, 0.8040589690208435, -0.340686172246933, 0.06181034818291664, 0.21841119229793549, -0.8218077421188354, 1.6725605746614747e-05, 1.96162748336792, -0.8963765501976013, 1.548850178718567, -0.5301591157913208, 2.7636892795562744, -1.071006417274475, -0.7261289358139038, -0.9822043776512146, 3.1507320404052734, 0.5644764304161072, 0.2897090017795563, -1.2915579080581665, -1.2476122379302979, 1.354856014251709, 0.8968766331672668, -0.14339196681976318, 0.31438469886779785, -0.4678103029727936, 0.7109728455543518, -0.0279061459004879, 0.7203503251075745, -1.0169464349746704, 0.5048996210098267, -0.2795088589191437, -0.22410041093826294, 1.5795000791549683, 1.4054040908813477, 0.5393300652503967, -0.4032829999923706, 0.23166857659816742, -0.009503702633082867, 0.12188363820314407, -0.03063902258872986, -0.6627312302589417, -0.8617584109306335, 0.9505792856216431, -0.27945733070373535, 0.5825731158256531, -1.4983609914779663, 0.38522183895111084, 0.26953914761543274, 0.32479235529899597, 2.260998487472534, -0.5797125101089478, -1.9842665195465088, -1.0890430212020874, 0.5012403130531311, 0.06120969355106354, -1.554655909538269, -0.2890305519104004, 0.573742151260376, 1.3245832920074463, 1.5962954759597778, -0.8179917931556702, 0.0636141374707222, -0.5796666741371155, -1.261069655418396, -0.7360519766807556, 0.3182908892631531, 0.8764958381652832, -0.4716455042362213, 1.2465450763702393, -1.2810379266738892, 0.5164843797683716, 0.6958470940589905, -0.5327696204185486, -1.3968422412872314, -0.555505633354187, -0.3471989631652832, -0.3470357656478882, 0.03540491685271263, 1.066373586654663, 1.442226529121399, -0.5142983198165894, 1.4413495063781738, 1.6024789810180664, -1.5107041597366333, 0.8112490773200989, 1.849635362625122, -0.7883493304252625, -0.41909077763557434, -1.3096497058868408, -0.7704923152923584, 0.6494381427764893, 0.5261595845222473, -2.7003440856933594, 1.2525752782821655, 0.6645267009735107, 0.4393843114376068, -0.4550272524356842, 1.6289507150650024, 0.11455994844436646, 0.38292914628982544, 0.6224557161331177, 2.0831174850463867, -1.0303031206130981, 0.7119752168655396, 1.5890976190567017, 2.815763473510742, 1.5950462818145752, 0.5282863974571228, -1.324933648109436, -1.0827430486679077, -1.519696593284607, -1.7325389385223389, -0.12465948611497879, 1.5212076902389526, -0.7441942095756531, -2.56120228767395, 0.009464937262237072, -0.4485029876232147, -0.3226776719093323, -0.3285345733165741, -0.4920157790184021, 1.6552330255508423, 0.3268011510372162, -0.31391382217407227, -0.3684370517730713, 1.6369833946228027, 0.629054605960846, -0.9137563109397888, -0.6671998500823975, -1.1535412073135376, 0.046469055116176605, -0.44814300537109375, 0.0492599718272686, 0.9324041604995728, 0.3289278447628021, -0.24206998944282532, -0.40933093428611755, -0.29863160848617554, 0.2900444269180298, 0.12190908193588257, -1.0838172435760498, -0.09108952432870865, 0.8236786127090454, 0.04154815524816513, -0.47531822323799133, -0.10074348002672195, -0.5369265675544739, -0.30831822752952576, -0.5750168561935425, 1.4824875593185425, -0.8158599138259888, -1.0211514234542847, -0.6757667064666748, -1.0111445188522339, -0.7750881910324097, 0.785151481628418, 0.2128245234489441, 0.03197348862886429, -1.059047818183899, 0.08576571941375732, 2.9785680770874023, -1.5573843717575073, -0.6702495813369751, -1.3982069492340088, 1.7582321166992188, 0.3829067051410675, -0.9910164475440979, 0.09221109747886658, 2.165250301361084, -0.24999591708183289, 1.3200715780258179, -1.0081393718719482, 0.5798066258430481, -0.846403956413269, -0.5747615098953247, 0.6670098900794983, 1.3025158643722534, -1.3715052604675293, -0.6218779683113098, -0.5797709226608276, -0.8282580971717834, 0.04487747699022293, 0.2466953545808792, -0.04343430697917938, 0.6842980980873108, 0.19478172063827515, 1.1822186708450317, -0.4067716598510742, 1.6332162618637085, -0.23470640182495117, -0.32707101106643677, 0.6818684935569763, -1.652695894241333, 0.2795220911502838, -1.1546552181243896, -0.3261899948120117, -0.7247691750526428, 1.5481703281402588, -0.219718337059021, 0.09158016741275787, -0.18381492793560028, -0.3668094575405121, 0.5687076449394226, 1.83064603805542, -0.014971893280744553, 0.2365511655807495, -0.5997337698936462, 0.9838067889213562, -2.486905097961426, 1.119157314300537, 1.5198200941085815, 0.01603871025145054, -1.1314104795455933, -0.06828288733959198, -0.19534853100776672, 0.45873117446899414, -0.9523457288742065, -0.7183239459991455, 0.47903773188591003, -0.45251375436782837, 1.65703284740448, 2.0574469566345215, 1.1895588636398315, -0.8584158420562744, 0.7636666297912598, 0.23211193084716797, 0.758985698223114, -0.7924146056175232, 1.4593149423599243, -0.6139424443244934, 1.8832536935806274, -0.595910906791687, 1.3653357028961182, -0.7907904386520386, -0.6482648253440857, 0.48557722568511963, 2.0967085361480713, -0.3492007553577423, -0.7437058687210083, 1.8860880136489868, 0.10805866122245789, 0.20203949511051178, 0.43306928873062134, -0.6878679394721985, -1.3146095275878906, -0.21555070579051971, 1.846340537071228, 1.315514087677002, 0.5665509104728699, 0.08488570898771286, -1.455124855041504, -0.20666955411434174, -0.5083174109458923, 0.46972790360450745, 0.2130368947982788, 0.5182521343231201, 0.6201753616333008, -0.2642947733402252, -0.9101539254188538, -0.01552516408264637, -0.35411226749420166, 1.1853026151657104, -1.0207773447036743, -1.5767338275909424, -1.9704617261886597, -0.23890917003154755, -1.610357642173767, 1.3292381763458252, 0.5344769358634949, -2.900686502456665, -0.14413560926914215, -1.7741299867630005, 0.38261058926582336, 0.7708978056907654, -0.17036594450473785, 1.8184118270874023, 1.087202787399292, 0.6559369564056396, 0.4409097731113434, -1.9196641445159912, 1.9613338708877563, 0.649265468120575, 1.552046537399292, 1.113482117652893, 0.06447534263134003, -1.1479898691177368, 1.362255334854126, -0.16428959369659424, -0.4953175187110901, 0.027567008510231972, 0.0038441282231360674, -0.48238787055015564, -0.2601609230041504, -0.008896308951079845, -0.709470808506012, 0.03745098039507866, 0.26604706048965454, -0.3318336606025696, 0.08326078951358795, -1.0423705577850342, -0.7108352184295654, -1.4972174167633057, -0.06638968735933304, -4.019217491149902, -0.08108017593622208, 0.5546515583992004, 0.1936901956796646, -2.5845587253570557, -1.2950347661972046, 0.5995382070541382, -0.18004043400287628, 0.3601052761077881, -1.5015947818756104, 0.027628440409898758, -1.0571237802505493, -0.9077930450439453, -0.5805213451385498, 1.3517552614212036, -0.9966403841972351, -0.29563024640083313, 0.4103727340698242, -0.1900159865617752, -0.006684183143079281, 1.3979278802871704, -0.32860836386680603, 0.3241075277328491, -0.7560264468193054, -1.990517020225525, -0.11167768388986588, 0.6040698289871216, -2.1756412982940674, -1.1858354806900024, -0.28401443362236023, 0.3539183735847473, 1.5428303480148315, 0.9522773027420044, 1.401229977607727, 0.39579859375953674, -0.9092941284179688, -0.2411983162164688, 1.482569694519043, 0.11510000377893448, 0.7538026571273804, -0.6916747093200684, 0.5595893263816833, -0.07639436423778534, -0.2759788930416107, 0.45829084515571594, 2.068620204925537, 1.3099373579025269, -0.14686496555805206, -0.6871467232704163, 0.31529751420021057, -0.18954706192016602, -1.2265998125076294, 0.09722165763378143, 1.2291030883789062, -0.5371788144111633, 1.2824982404708862, 1.3511654138565063, -0.3918490409851074, -0.7137337923049927, -0.5214012265205383, -0.4994742274284363, 1.25232994556427, 0.693888247013092, -0.22451499104499817, 0.42081472277641296, -1.33063805103302, 0.3920499384403229, -0.8769280314445496, -0.09512033313512802, 0.4842071533203125, 0.5746899247169495, -1.2580260038375854, -0.2819386124610901, -0.7713513970375061, -0.7628244757652283, 0.8857622742652893, -0.33091169595718384, 0.020017361268401146, -1.2448111772537231, -0.18664227426052094, -2.006221055984497, 1.6929447650909424, -0.2208370864391327, -0.5446068048477173, -1.61741304397583, 0.3481315076351166, 0.21314716339111328, -0.7491135001182556, 0.6886699199676514, -0.37928083539009094, 0.5570600628852844, -0.3334845304489136, 0.7198548913002014, 0.6719112992286682, -0.8508391380310059, 1.2864470481872559, 1.092836618423462, -1.2277945280075073, -0.3573976159095764, -0.06433826684951782, -0.41162797808647156, 0.9305658936500549, -0.3792159855365753, -1.8557446002960205, 0.26365894079208374, -0.7472548484802246, 0.657619059085846, 1.4531484842300415, 0.5198113322257996, 0.1478559821844101, -1.2982208728790283, 0.8955221176147461, -0.008366267196834087, -0.2350229024887085, -1.058437705039978, 1.5397895574569702, -0.41632869839668274, 1.1841175556182861, -0.5541256666183472, -1.6504626274108887, -1.384183645248413, 0.07745006680488586, 0.7101803421974182, -0.7809345126152039, 0.8537569642066956, 0.31728795170783997, -0.23426377773284912, -1.9010869264602661, -0.6056231260299683, 0.93621426820755, -0.4221937954425812, -0.2082713395357132, -1.0942169427871704, 0.027934104204177856, -1.9274710416793823, 1.4884332418441772, 0.1721191555261612, 0.9949748516082764, -1.4233766794204712, 0.0905982255935669, 0.6055582165718079, -0.36803537607192993, 0.2661173641681671, 0.5057647824287415, -1.4021040201187134, -0.6565370559692383, -0.36518585681915283, 0.4337281584739685, -0.6311211585998535, -0.10307621955871582, 0.31935498118400574, -0.21411845088005066, 1.370286226272583, 0.9950563311576843, 0.05073494464159012, 0.7587189078330994, -0.4041997790336609, 0.4306054413318634, -0.021252496168017387, -0.25654593110084534, -0.017595142126083374, -0.1101345419883728, -0.5265159010887146, 0.0019304861780256033, 0.5087857842445374, 1.8476577997207642, -0.2491852343082428, -0.06533169001340866, 0.8284899592399597, 0.4625686705112457, 0.6502141356468201, 0.7030585408210754, -0.11501173675060272, 1.0409648418426514, 0.3475346267223358, -0.9225931763648987, -0.004013114143162966, 1.1659762859344482, -0.6312037110328674, -0.9200157523155212, -0.2669219970703125, 0.03868020698428154, -0.5943773984909058, -1.1208381652832031, -0.22489134967327118, 0.031682875007390976, 0.2766953706741333, -0.8561738133430481, 1.358408808708191, 1.0662829875946045, -1.366140604019165, -1.7710866928100586, 2.480804204940796, -1.1961643695831299, -0.3454311192035675, -0.061730772256851196, 0.6307605504989624, -1.0722960233688354, -1.4426935911178589, 1.0223252773284912, -0.7275472283363342, 0.7521810531616211, -0.7191532254219055, -0.17688274383544922, -1.0166215896606445, -0.5772945284843445, 0.2054869532585144, 0.2739531695842743, -0.2267276495695114, 0.282262921333313, -1.035406231880188, -0.1976625621318817, 0.4814082682132721, -0.5063855648040771, -0.9106087684631348, -0.18691900372505188, 1.2334206104278564, 1.3447139263153076, -1.8331736326217651, -0.7159862518310547, -1.687506079673767, -0.23338064551353455, 0.21402275562286377, 0.4726313650608063, -0.0056795342825353146, 0.08040884137153625, -0.558793306350708, -0.3149837851524353, -0.4539733827114105, -0.6408160924911499, 0.7562175989151001, -0.22089450061321259, -0.44891393184661865, 0.3162313401699066, -0.26069170236587524, 0.3677920401096344, -1.421934962272644, -0.31421101093292236, 0.2558140158653259, 0.24202612042427063, 0.2020941525697708, 1.4966665506362915, -0.667299747467041, -0.6212731599807739, 0.7388451099395752, -0.7916216254234314, -1.6735118627548218, 0.30964401364326477, 0.19086365401744843, -1.9777158498764038, -1.218419075012207, -0.736964762210846, 0.5467699766159058, 0.32153621315956116, 0.4797658920288086, 1.2632100582122803, -0.5797511339187622, 0.7367792725563049, -0.47786134481430054, -0.1680062711238861, 1.2991986274719238, 0.39390799403190613, 0.5249650478363037, 0.5061703324317932, 0.1781385838985443, 1.0269955396652222, 0.3759840726852417, 0.08054256439208984, -0.12461249530315399, -1.296888828277588, 0.3933161795139313, 2.0249686241149902, -0.28426074981689453, -0.669919490814209, -1.1766481399536133, -0.33469319343566895, -0.08139923214912415, 0.6560235619544983, -1.0964058637619019, -0.09862225502729416, -1.5406181812286377, -1.509379506111145, -1.3236734867095947, 1.6520106792449951, 0.4647139608860016, -0.7038848996162415, 1.6172689199447632, 0.9980254769325256, 0.1519325226545334, 0.5447366833686829, -0.73585444688797, 0.45309552550315857, -0.9439312815666199, -0.722134530544281, 1.1171362400054932, -0.9354068636894226, 2.046074867248535, -1.111352562904358, -0.6997200846672058, -0.2877329885959625, -0.15632718801498413, 0.6085320711135864, -0.2324460744857788, -1.1726515293121338, -0.44040361046791077, 1.2939400672912598, 1.1165592670440674, 1.4386497735977173, -0.7262520790100098, 1.449089765548706, -0.6631131768226624, -0.6763084530830383, 0.33402809500694275, -0.640042781829834, -1.000062108039856, 0.38267192244529724, -0.0036812203470617533, -0.1588592529296875, 0.5138590335845947, -0.15400706231594086, 0.03481220453977585, 2.2424325942993164, -0.6312185525894165, -0.48035719990730286, 1.6814104318618774, 0.2641405463218689, -0.43469947576522827, -0.4749261736869812, -0.10427287220954895, 0.23619233071804047, -0.11381374299526215, -0.8424742221832275, -1.6419636011123657, 0.740386962890625, -0.5067701935768127, 1.4950028657913208, 0.508384644985199, -0.6438630819320679, -0.6875940561294556, -0.6774080991744995, -0.406807005405426, -0.7212463021278381, 1.2705440521240234, -0.026512641459703445, 1.2645697593688965, 0.12506689131259918, -2.0195751190185547, 0.07703378796577454, -2.1639394760131836, -1.1398146152496338, -0.3462502956390381, 0.42113813757896423, -1.2450379133224487, -0.6621242761611938, -0.9319572448730469, -0.31765151023864746, -0.8525834679603577, 0.22911103069782257, -0.5362128019332886, 0.39558136463165283, -0.2406405806541443, -1.2342780828475952, 0.31486549973487854, 0.72835373878479, 0.43725937604904175, -0.8684770464897156, 1.036826491355896, 0.8842031955718994, 0.5806310772895813, 1.3760645389556885, 1.2813916206359863, 0.7104974389076233, -1.0837328433990479, 1.2458720207214355, 0.0010351191740483046, -0.3681407570838928, 0.25882381200790405, -0.5553528070449829, -0.15150173008441925, -0.6618583798408508, 1.6541290283203125, 0.8182324171066284, -1.2120046615600586, 1.7986575365066528, -2.2134816646575928, 0.27697432041168213, 0.8679805397987366, -1.6715795993804932, -0.5538877844810486, -2.146103620529175, -0.3205223083496094, -2.2725703716278076, -0.8686763644218445, -0.2887036204338074, 0.0949544683098793, -0.6548179388046265, -0.4383220970630646, 1.4037423133850098, -0.6476320624351501, -0.7418309450149536, 1.5449148416519165, -0.20134983956813812, -0.24899640679359436, -1.1426684856414795, 0.7779062986373901, 0.8571075201034546, 0.8938401341438293, 0.6636249423027039, 0.18894042074680328, -2.0111255645751953, -1.498239517211914, 0.01817956753075123, -0.46423816680908203, 1.510159969329834, -0.07176920771598816, -0.03444176912307739, 0.4424111843109131, -2.161773443222046, -0.792534589767456, -2.231673240661621, -0.4375583529472351, 0.13570180535316467, -0.9643506407737732, 0.40542834997177124, -0.5053149461746216, -0.12080997228622437, -0.3575649559497833, -0.0729093998670578, 1.5619182586669922, -2.642199754714966, 0.7052647471427917, -1.5336780548095703, 1.2993698120117188, -0.07576248794794083, -0.7056576013565063, -0.4780043065547943, -1.4571748971939087, -0.9819554090499878, 1.2965986728668213, -1.320007562637329, -0.4199678301811218, 1.9746317863464355, -0.4021122455596924, -1.1435887813568115, 0.4366588592529297, -0.06468295305967331, 1.1854300498962402, -0.10853855311870575, 1.024337887763977, 0.08258681744337082, -0.06065179035067558, -0.9407694339752197, 0.7012963891029358, 1.0797301530838013, 0.304524689912796, -1.1821497678756714, -1.049230933189392, 0.5186152458190918, -0.5419479608535767, -0.18852630257606506, -1.2926795482635498, -0.283313125371933, 0.738954484462738, 0.1662192940711975, -0.9929981231689453, -0.7991530299186707, -0.8569441437721252, -0.6315046548843384, 1.0085278749465942, -0.4699346125125885, -0.09507905691862106, 0.01350745651870966, 1.5803728103637695, -1.1744593381881714, -0.8764644861221313, -1.053227424621582, -1.0012845993041992, 0.8223708868026733, -0.28259924054145813, 0.38864070177078247, 0.7361662983894348, 0.46160754561424255, -0.7813830375671387, -0.02459063194692135, 1.2716470956802368, 0.11588191241025925, -0.04023623466491699, -0.870347797870636, 0.9433103799819946, -0.5118637084960938, -0.3693423271179199, -0.7104698419570923, 0.9858725666999817, 1.4065181016921997], [-1.0611391067504883, -0.9218658208847046, -1.6297527551651, 0.08581872284412384, 0.7081043720245361, -0.32288476824760437, -0.7863813638687134, -0.8669739961624146, 0.5519104599952698, -0.5506616830825806, 1.7527621984481812, -1.19808030128479, -0.6980224847793579, -1.6817057132720947, 0.13554206490516663, 0.7261933088302612, 1.0535141229629517, -0.4601481854915619, -0.36875149607658386, -0.7986077666282654, 0.9343522191047668, 0.5863886475563049, 0.005563313607126474, 1.4025579690933228, -1.4269568920135498, -1.3341213464736938, -1.5711637735366821, -1.7971599102020264, -0.5345736145973206, 2.0734004974365234, 0.21423736214637756, -0.5985114574432373, 0.8512714505195618, -1.9269254207611084, -0.13890308141708374, 1.3317266702651978, -0.3008086085319519, -0.8156348466873169, -0.5068808197975159, 1.0876296758651733, 0.4816409647464752, -0.3226696252822876, 0.33086928725242615, 0.37530046701431274, -0.5867221355438232, -0.48537570238113403, -0.8845695853233337, 0.5188039541244507, -2.2849719524383545, -0.6931084990501404, 1.3962737321853638, 1.2075269222259521, 0.05996985733509064, -0.405376136302948, -2.016240358352661, 1.202721118927002, 0.5322562456130981, -0.7631328701972961, 0.8160564303398132, -0.44752874970436096, -1.5044136047363281, -0.4612114429473877, 1.1027077436447144, -0.9753378033638, 0.5575544238090515, 0.6099338531494141, 0.1948702186346054, -0.1638426035642624, -0.6511725187301636, -1.5480444431304932, 1.327101469039917, -1.4303687810897827, 0.26585033535957336, 1.0022436380386353, -0.41939038038253784, -1.9119846820831299, -2.7600693702697754, -1.1115261316299438, -1.2201130390167236, 0.037607330828905106, -0.10501013696193695, -0.5660484433174133, -0.8091903328895569, 0.12096603959798813, 0.32145681977272034, 1.4443625211715698, 0.756672203540802, -1.5558232069015503, -0.04794308915734291, -0.1549173891544342, 0.3149154782295227, 0.2201145440340042, -1.4469196796417236, -1.9847952127456665, 0.07634570449590683, -1.6311357021331787, -0.5518749356269836, -0.6052095293998718, -2.032057046890259, 1.149928092956543, -2.408224582672119, 0.5468018651008606, 0.9334746599197388, -0.37404847145080566, 0.9888190031051636, 0.2392977923154831, -0.3760983943939209, -0.2323307991027832, -0.3512319028377533, 1.8083642721176147, -0.6843764781951904, -0.05860745161771774, -2.246342897415161, 0.20919181406497955, 2.4320456981658936, -0.9859733581542969, -0.22142450511455536, 0.055845268070697784, -0.13201095163822174, -0.46293121576309204, 0.2899312973022461, -1.149159550666809, -0.48752444982528687, -0.5267953276634216, 0.37711185216903687, 0.22290407121181488, -0.297822505235672, 0.43235406279563904, -0.09520262479782104, 0.9292359948158264, 0.4237239956855774, -0.11343662440776825, -0.1753617376089096, -0.6724324226379395, -1.2758655548095703, 1.6827502250671387, 0.5961016416549683, -0.913484513759613, 0.5134121775627136, 1.644716501235962, -0.8318492770195007, -0.0341278575360775, -0.3965906798839569, 2.041071653366089, 0.39564692974090576, 0.6132848262786865, -1.2695953845977783, 0.9294010996818542, -1.4078032970428467, -0.2739485502243042, -0.9745035171508789, 0.41011321544647217, 1.1779630184173584, -0.626995861530304, -0.5604014992713928, -0.8815555572509766, 0.8370348811149597, -1.8018325567245483, -1.7206850051879883, -2.0946462154388428, -1.3825682401657104, -0.5597429275512695, 0.37753525376319885, -0.663982093334198, 0.009094799868762493, 0.11655943840742111, -0.04782598093152046, 0.14899444580078125, -2.1232852935791016, 1.8235808610916138, 0.7613193392753601, -0.5896871089935303, 1.647564172744751, -0.0758683905005455, 1.4105850458145142, 0.9195148348808289, -0.17790618538856506, 2.022066593170166, -2.077819585800171, -1.1924179792404175, -0.9596698880195618, -1.632107138633728, 1.1155654191970825, -1.4991505146026611, -0.6596734523773193, -0.2220255583524704, 0.937088131904602, -1.0708547830581665, -1.598926305770874, -0.31722694635391235, -0.21058742702007294, 0.0979098379611969, 1.5399541854858398, -0.2503686845302582, 0.8947675824165344, -0.4383379817008972, 1.4451031684875488, -1.5947524309158325, -0.4523407518863678, -0.7312119007110596, -0.8837070465087891, 0.8660562634468079, -0.4744355082511902, -0.010382580570876598, -1.3248587846755981, -1.2966536283493042, -0.253559947013855, -0.6634171605110168, -0.9310770034790039, 0.27284032106399536, 0.3481062352657318, -0.5651827454566956, 0.027237018570303917, 0.31729933619499207, 1.0939549207687378, 0.8456516265869141, 1.3310974836349487, 2.3758766651153564, -0.9920207262039185, -0.10268763452768326, -1.3371081352233887, 1.1981874704360962, -1.2976126670837402, -1.69783353805542, 0.43581557273864746, -0.6275639533996582, 1.9437026977539062, 0.26826563477516174, -0.0013402481563389301, -0.19963613152503967, -0.1512690931558609, -1.9663275480270386, 1.0040628910064697, -1.0455095767974854, 0.23576800525188446, 0.09689435362815857, -1.5311908721923828, -0.6700196266174316, 0.945358395576477, -0.4564545750617981, -1.4108198881149292, 0.31691688299179077, -0.6408951282501221, -1.702187180519104, -0.34541094303131104, -0.41258877515792847, 0.6936687231063843, -1.1623655557632446, -1.5864264965057373, 1.0454994440078735, 0.14221936464309692, 0.14789515733718872, 0.3507990837097168, 0.2318452000617981, -0.40013667941093445, -1.7146694660186768, 0.2379937618970871, 0.8379818797111511, -0.03465058654546738, 0.1461932212114334, -0.8910183310508728, 0.6157699227333069, -0.8760175108909607, -0.6265621185302734, 0.396122008562088, -0.25250503420829773, -0.16175638139247894, 0.5592396259307861, -0.01820327900350094, 1.5494812726974487, 0.10104138404130936, 0.8177096843719482, 0.4715564250946045, 0.08213889598846436, -0.45317280292510986, -0.807975172996521, -0.8185022473335266, -0.7984318137168884, -1.6471912860870361, 0.9207586050033569, -0.6328694224357605, -0.25398921966552734, -0.4637306034564972, 0.3748297691345215, 0.14069873094558716, 1.2834582328796387, -1.3827861547470093, 2.7907276153564453, 1.0972952842712402, 0.11506074666976929, 0.4100002646446228, 0.5018125176429749, -1.0449535846710205, -0.34488359093666077, 0.27866026759147644, 0.007815585471689701, 0.3962780833244324, 0.33003896474838257, 0.10878659039735794, 0.3996429443359375, 0.4872550070285797, -1.5153876543045044, -0.5853300094604492, 1.518031120300293, 1.1491590738296509, 0.23583687841892242, 1.5493519306182861, -0.1776527762413025, -1.3882886171340942, -0.870369017124176, 0.40602758526802063, 0.27593880891799927, 0.3438165783882141, -0.34654152393341064, -0.787517249584198, -0.061024393886327744, 0.08076129108667374, -1.30631422996521, -1.307598352432251, 2.247772693634033, 2.167048692703247, 1.114489197731018, -0.4800160527229309, 0.20403192937374115, -1.7892670631408691, 0.15325450897216797, 0.7279906272888184, 0.0883159190416336, 0.7289339900016785, 0.039214324206113815, 1.5428946018218994, 1.0444387197494507, -0.2064414918422699, -0.005898655392229557, 0.007635646499693394, 0.7343085408210754, -1.0495728254318237, 1.385221004486084, 0.7462679147720337, -0.13746844232082367, 2.4923012256622314, -0.3053441047668457, 0.2884354889392853, -0.7084094285964966, 0.673670768737793, -1.357736587524414, 1.296087384223938, 0.6605990529060364, -1.956520438194275, -0.4414461851119995, 0.6882225275039673, 0.09335365146398544, 0.06447670608758926, -0.15723225474357605, -0.7384673357009888, -1.8535221815109253, 1.1831464767456055, 0.08414293080568314, -0.5631385445594788, -0.9905572533607483, -0.1582852602005005, -0.8544703722000122, 0.6997506618499756, -1.8847588300704956, -0.5881904363632202, 1.336696743965149, -0.9516133069992065, -0.17768387496471405, -0.7442892789840698, 1.0199778079986572, 1.3683122396469116, 0.707804262638092, -0.5367679595947266, -1.6852158308029175, 0.9890341758728027, 0.1549162119626999, 1.2586408853530884, 2.8627216815948486, -0.27021265029907227, -0.3505916893482208, -0.579210102558136, -0.11471537500619888, -0.6593749523162842, 1.124610424041748, -0.8344659805297852, 0.4295431077480316, -0.8876226544380188, -0.8321338295936584, 0.46480241417884827, -0.9150316119194031, -0.6097903251647949, 1.1147955656051636, -0.7889083027839661, 1.1307512521743774, -0.5746669769287109, 1.0076539516448975, -1.4002975225448608, 0.8126740455627441, 0.14102903008460999, -1.3743987083435059, 1.4049164056777954, -1.0224549770355225, -0.8567435145378113, -0.31419268250465393, 0.17238463461399078, -1.6062939167022705, -0.2943874001502991, 0.5530267953872681, 0.5042343139648438, 0.1500217467546463, -0.6404516696929932, 0.940054178237915, 0.4246533513069153, 0.5150041580200195, 0.5815555453300476, -1.0787395238876343, -1.1882303953170776, -0.2575733959674835, -1.4266456365585327, 1.7481039762496948, -0.40489086508750916, 0.8097022771835327, 1.503087043762207, 0.08476805686950684, 0.4863629639148712, 2.971787929534912, -0.02650311402976513, -0.7417515516281128, -0.8159913420677185, 0.0727144405245781, 0.7558276057243347, 1.8240751028060913, 0.20749454200267792, 1.502298355102539, -1.1063088178634644, 0.0031277849338948727, -0.25712066888809204, 0.527518630027771, -0.4964234232902527, 0.0667620375752449, -1.1358543634414673, 1.0521845817565918, 0.04410090669989586, -1.6547552347183228, -1.3965177536010742, 1.0263036489486694, -2.210089921951294, 2.045938491821289, 0.32842743396759033, 0.9432482719421387, 0.8516773581504822, 1.882156252861023, -1.7335631847381592, 0.19957119226455688, -0.4706019461154938, 1.0665608644485474, 0.5346935391426086, -1.7171030044555664, -0.7348865270614624, 0.5678723454475403, 0.1896129846572876, 0.5800459980964661, -0.7581272125244141, -0.20920445024967194, -1.784859538078308, -0.9606802463531494, -0.23722094297409058, 1.4700310230255127, 1.2477415800094604, -1.3231765031814575, -0.5815497636795044, -1.7282047271728516, 1.1774101257324219, 1.1851577758789062, 0.15654690563678741, 0.3050256371498108, -1.455084204673767, 1.00543212890625, 0.5620025992393494, 1.603392243385315, -0.5027084350585938, 0.00012533232802525163, -0.6174514293670654, -1.1403720378875732, -0.6636368632316589, -2.3961246013641357, 1.9170491695404053, -0.37575432658195496, 0.6322875618934631, 0.38984185457229614, -1.2844741344451904, -1.0762492418289185, 1.3399056196212769, 1.5508944988250732, 0.2023734450340271, 0.2789544463157654, -1.107806921005249, 0.4150031805038452, -1.5343258380889893, 0.9379066824913025, -1.0258219242095947, -0.9526246190071106, -0.9826822280883789, -1.2661395072937012, -0.7757465839385986, 0.523128867149353, -0.28349798917770386, 1.148606538772583, -0.2850095331668854, -0.2627404034137726, 0.38270172476768494, -0.6400620937347412, 0.12028040736913681, 1.46332585811615, -1.7230576276779175, -0.985386312007904, -1.2805581092834473, 0.16955256462097168, -1.0706952810287476, -0.0026183954905718565, -1.6150187253952026, 1.2029284238815308, 0.12852177023887634, -0.07710518687963486, -0.02383827604353428, 0.40728211402893066, -0.04854602366685867, 0.5822457671165466, -0.7897688150405884, 0.2693566083908081, -2.1080894470214844, -0.4228873550891876, 0.18816061317920685, -1.2140403985977173, -0.9287305474281311, 0.3374082148075104, 0.2229638546705246, -0.9665683507919312, 1.0718995332717896, 0.46813321113586426, -0.7650104761123657, 0.7535024881362915, 0.5749737024307251, 0.5393124222755432, -0.8776142001152039, -2.0303924083709717, 0.7127503752708435, 0.03537275269627571, 1.4351773262023926, 0.6454965472221375, 0.4528863728046417, 1.4490395784378052, 0.3400050699710846, -0.9710760116577148, -0.71148282289505, 0.9683139324188232, 2.1676859855651855, 0.5943309664726257, -1.7100855112075806, -0.4052531123161316, -0.45570898056030273, 1.0456775426864624, -0.7049583792686462, -0.06722141802310944, -1.043854832649231, 1.116331934928894, -0.08136188238859177, 0.27694177627563477, 0.032843053340911865, 0.9496583342552185, -0.4050653874874115, -0.23892778158187866, 1.1864246129989624, -0.7448344230651855, -0.5653839707374573, 0.01463681273162365, 1.1645079851150513, 1.47519850730896, -0.8278334140777588, 0.5310871601104736, -0.041343409568071365, -0.4503541886806488, -0.06648218631744385, 1.3191574811935425, -0.29653826355934143, 0.987656831741333, -0.5028550624847412, 0.7735708951950073, -0.4852428734302521, -1.1822516918182373, 0.25224751234054565, -1.4638956785202026, -1.4774278402328491, -0.3483331799507141, 0.08787143230438232, 0.23472361266613007, 0.7612609267234802, -1.3079057931900024, -0.8134818077087402, -1.0687581300735474, -0.8116045594215393, 0.4548538029193878, 0.3529164791107178, -2.017571449279785, -0.6561013460159302, -0.3302711248397827, 0.9664225578308105, 0.39882054924964905, -0.21413427591323853, 0.07437891513109207, 0.033319562673568726, 0.8046485781669617, 0.6319341063499451, -1.1164695024490356, 1.5572718381881714, 0.563900887966156, 1.163251519203186, 0.7249001860618591, 0.3469102680683136, 0.1565741002559662, 1.1461902856826782, 0.1163364052772522, 0.9488812685012817, -1.15031099319458, 0.029380083084106445, -1.2663235664367676, -0.6101862788200378, 0.6909975409507751, 0.8829782605171204, 1.655672311782837, 1.1679216623306274, 0.434758722782135, 1.6805630922317505, 0.6259691119194031, 0.30560874938964844, 2.239607810974121, -0.06130818650126457, 1.315671443939209, -1.1257954835891724, -0.7166646718978882, 0.2892257571220398, 1.6102153062820435, -0.4289976954460144, -1.5339463949203491, 1.2587926387786865, 0.6834267377853394, -0.5433180332183838, 0.03904351964592934, 1.0220696926116943, -0.4650380313396454, -1.1385712623596191, -1.1164865493774414, 0.6595194935798645, -0.5094809532165527, 1.4346919059753418, -0.5814183354377747, -1.5173251628875732, 0.39322593808174133, -1.22548508644104, 1.2431671619415283, -0.3981509804725647, -0.04421592131257057, 0.5147465467453003, -0.10620024055242538, -1.7908848524093628, 0.02611977979540825, 0.14998067915439606, -0.6299090385437012, -1.4375402927398682, 0.12452825903892517, -0.6133062839508057, -0.15022003650665283, -0.9071441888809204, 0.22937650978565216, 0.18747854232788086, 0.5138057470321655, -0.982409656047821, 0.6527596712112427, 0.10392072796821594, 0.23102501034736633, -1.2408276796340942, -0.7225167751312256, -0.3272321820259094, -1.6115983724594116, 0.41922029852867126, -1.8806798458099365, 0.416886568069458, 1.4560941457748413, 1.2761180400848389, 1.1271418333053589, -0.9548062682151794, -1.8195571899414062, -0.21993625164031982, -1.1261180639266968, 0.5869024991989136, 0.14558257162570953, -0.45916011929512024, 0.316739022731781, -1.3805787563323975, 0.6810813546180725, 1.1274534463882446, -1.2969670295715332, -0.48009225726127625, -0.0381409227848053, 0.8345771431922913, -0.7630926370620728, 1.3192485570907593, 2.7947628498077393, 1.2758913040161133, 1.3278226852416992, -0.4109702706336975, -0.04149269312620163, -1.8712223768234253, -0.2852320969104767, 1.1138676404953003, 0.21059425175189972, 0.8007352352142334, 0.7764772176742554, 2.03066349029541, -0.11388710141181946, 1.0182589292526245, 0.3387118875980377, 1.831257939338684, 0.8460195660591125, 1.2684184312820435, -0.14139336347579956, 1.6159353256225586, -0.5172328352928162, -2.204314947128296, -0.5881324410438538, -1.3310822248458862, -1.4285463094711304, -0.8383135795593262, 0.8762107491493225, -0.2350398451089859, -1.2934805154800415, -0.5361547470092773, -0.8517218828201294, -0.6628046631813049, -0.8603206872940063, 0.2595817744731903, 0.5041323900222778, -0.12434088438749313, 0.8206826448440552, -1.6024489402770996, 1.8145642280578613, -0.5130056142807007, -0.42583364248275757, -1.3246338367462158, -0.10559246689081192, 1.6468502283096313, 1.1460024118423462, 0.1459006369113922, 0.1249055489897728, -0.505991518497467, -0.3043193817138672, 0.6747921109199524, 0.994479238986969, -0.3891308605670929, -0.3388575315475464, -0.7069219946861267, -0.5846356153488159, -0.6406579613685608, -2.2423458099365234, -0.847084641456604, 1.2276862859725952, 0.15552137792110443, 0.9042606949806213, -0.06001240015029907, -0.7595875859260559, 0.07128220051527023, 1.2500545978546143, 1.563147783279419, 1.7443253993988037, 0.16635853052139282, -1.918364405632019, 0.5950173139572144, 0.3017812669277191, 0.022367941215634346, 0.20775967836380005, -0.02101892977952957, 0.2379751205444336, 0.6932722926139832, 1.3958603143692017, 0.38396310806274414, 0.1331886649131775, 0.10532233119010925, 0.8449798226356506, 1.1884019374847412, -2.6193580627441406, 1.1962555646896362, 1.066778302192688, -1.4805008172988892, 0.7072869539260864, 0.21023622155189514, 0.9421981573104858, -0.8074894547462463, -0.9158686995506287, 0.21865904331207275, 0.25052517652511597, -0.8742710947990417, -2.153634548187256, -0.304070383310318, 0.19404086470603943, 1.9108635187149048, 0.18363921344280243, -1.7001852989196777, -0.7003677487373352, -0.3869582414627075, -1.5619714260101318, -0.5102218389511108, 1.3369743824005127, -1.434928297996521, -0.7027642130851746, -0.5325047373771667, -0.502117931842804, 0.7757757902145386, 2.367691993713379, 0.12760935723781586, 0.5690493583679199, -1.4219638109207153, -1.228617548942566, -0.11641344428062439, 0.35096463561058044, -0.28931450843811035, -0.26929691433906555, 0.1997942328453064, -1.1467268466949463, 0.6910977363586426, -0.13798923790454865, 1.1725964546203613, 1.7652515172958374, -0.387801855802536, -0.7011004686355591, -0.985893189907074, 0.8955059051513672, 0.5443877577781677, 0.3020578920841217, 0.9057996869087219, -0.20209600031375885, 0.19077125191688538, -0.23916076123714447, 2.8716132640838623, 0.7319080829620361, 0.5176233649253845, -1.4403119087219238, -0.9352062344551086, 0.30484047532081604, -0.6472859382629395, -0.257926881313324, -0.7113939523696899, -1.2548444271087646, -0.7507052421569824, 0.812090277671814, -1.2421058416366577, -0.7877139449119568, -0.27166739106178284, -0.5045544505119324, -1.4870553016662598, -0.500510573387146, 0.9528002738952637, -2.5693178176879883, -3.4836325645446777, -0.7554656863212585, 0.1428813487291336, -0.3140840232372284, 0.07037544995546341, 1.0794482231140137, 0.9076650738716125, 1.2283283472061157, -0.7912608981132507, -1.740968108177185, -0.5656607747077942, -0.2424216866493225, 1.2317460775375366, 1.8988755941390991, -0.4314548373222351, -1.4484336376190186, -0.18450087308883667, -0.36497849225997925, -0.23256047070026398, 0.8857369422912598, 0.9035967588424683, 1.1976265907287598, -0.37864798307418823, 0.08492662757635117, -0.3965981602668762, -0.7667186260223389, 0.4543898403644562, 0.8136390447616577, -0.05716504529118538, 1.6590324640274048, -1.1633780002593994, 1.733751654624939, -1.7755674123764038, -0.22071947157382965, -0.2774193584918976, -0.5969827175140381, 0.5870711803436279, 0.6592503786087036, -0.2759217917919159, 0.08032083511352539, -0.5308598279953003, 0.09917213022708893, -0.5885195136070251, 1.2461191415786743, -0.4031735360622406, -2.769846200942993, -0.27292707562446594, -0.3184259533882141, -1.3687081336975098, -0.27061033248901367, 0.9233108758926392, 1.1173932552337646, -0.4002762734889984, -0.4412487745285034, -0.07971777021884918, -0.8533729910850525, 0.09108905494213104, -2.33526349067688, -0.1785644292831421, -0.33607855439186096, 1.139154314994812, -2.5559158325195312, -0.9601559042930603, -0.7888789176940918, -1.0836589336395264, -0.0985308513045311, -0.2459615170955658, 0.36952537298202515, 0.9463237524032593, 0.4253399670124054, 0.1381663978099823, -2.6457762718200684, 0.5276575684547424, 1.7442808151245117, -0.45945772528648376, 2.3407700061798096, 1.7564525604248047, 0.540166974067688, 0.6962882876396179, 1.5172842741012573, 0.25674596428871155, -0.9220419526100159, -0.08894544839859009, -0.9277412295341492, 0.2255137711763382, -1.3298250436782837, -1.2520298957824707, 0.5764802098274231, -0.6371362209320068, -0.4196891188621521, 1.5973981618881226, 0.5112245678901672, -1.318294644355774, -0.03580646961927414, 0.25477835536003113, 0.8333892226219177, 1.175923228263855, -0.34992220997810364, -0.3839419186115265, -0.8253783583641052, -0.12784279882907867, -0.5270584225654602, 0.664117693901062, -1.8248350620269775, 0.4863578975200653, 0.6244807243347168, 0.579769492149353, 0.1588628739118576, 1.6188297271728516, -0.8130568265914917, 2.002875328063965, 0.8867833614349365, -0.14248418807983398, -0.003061338560655713, -0.5270101428031921, 0.5627372860908508, 0.10983846336603165, -1.6036280393600464, -1.2033544778823853, -1.1206088066101074, 1.3834813833236694, -0.016147878021001816, -1.2231395244598389, -1.2222903966903687, -0.298159658908844, 1.5448287725448608, -0.3580392301082611, -0.045227885246276855, 0.6901788115501404, -0.12940780818462372, 0.25044381618499756, -0.6814519166946411, 0.32725316286087036, 0.881887674331665, -0.37407010793685913, -0.8524243235588074, 2.0174190998077393, 0.008264715783298016, 0.28018179535865784, 0.04983489215373993, -0.12491224706172943, -0.5462812781333923, 0.36569342017173767, 2.091524362564087, 0.5116076469421387, -0.9411154389381409, 0.35610800981521606, -1.277247667312622, -3.2251431941986084, 0.9436724781990051, -1.1548694372177124, 0.07371336221694946, 0.1226596012711525, -0.8671169877052307, -2.453795909881592, 1.9622457027435303, -0.3201006054878235, -1.7989362478256226, -0.3290236294269562, 1.9262515306472778, -1.2821248769760132, -0.09677846729755402], [-2.1550538539886475, 2.3845839500427246, 0.8066347241401672, -0.11448603868484497, -1.6408073902130127, -0.4590061604976654, -0.8110738396644592, -1.5666660070419312, -0.7503781914710999, -2.328565835952759, 1.7672053575515747, 1.6061513423919678, -0.8852892518043518, 1.0494589805603027, -0.01562909595668316, 0.11662482470273972, 1.156781792640686, -0.3284416198730469, 0.6079365611076355, -0.30989301204681396, -0.7770350575447083, 0.5151717066764832, 1.4736850261688232, 0.6064222455024719, 1.8136446475982666, 0.025365643203258514, 0.6857278347015381, 0.015074169263243675, -0.5448347330093384, 0.004607747308909893, 1.6080478429794312, 0.597592294216156, 1.304885745048523, 0.8219230771064758, -2.4609858989715576, -0.059428922832012177, -0.07196784764528275, -1.2034937143325806, -1.324927568435669, -0.14523422718048096, -0.8849328756332397, -0.33240094780921936, -1.6417016983032227, 1.4308069944381714, -0.042080119252204895, -0.9518910646438599, -1.1800602674484253, -1.2630884647369385, 0.7822812795639038, 0.6608547568321228, -1.0191370248794556, 0.1962077021598816, -0.677006185054779, 0.4073515236377716, 0.3749171793460846, -0.21206152439117432, -1.4793645143508911, -0.2825491428375244, 0.08646184951066971, -0.021236274391412735, 1.0159673690795898, -0.4029850363731384, -2.189610242843628, -0.3027479350566864, 1.585563063621521, -1.510884404182434, -1.9247075319290161, -0.7521530389785767, -0.1930568367242813, -2.3171377182006836, -0.5674040913581848, 1.134527564048767, 0.9902681112289429, -0.17219296097755432, -1.033350944519043, 0.6927322149276733, 0.05469042807817459, 2.7789196968078613, -0.6590696573257446, -0.39548152685165405, 0.38865166902542114, -0.3878649175167084, -0.40820860862731934, 0.3873344361782074, -0.7918773293495178, 0.4079118072986603, 0.6472873091697693, 1.035524606704712, -0.5664631128311157, 0.004916857462376356, 0.241829976439476, -0.3642059564590454, -0.5936399698257446, -1.9903424978256226, 0.41555967926979065, -0.47211354970932007, 1.539351463317871, 0.9985837936401367, 0.5771923661231995, 0.6135861277580261, -0.9516910314559937, -0.5407643914222717, 0.8398036956787109, -1.09818434715271, 0.44921356439590454, 0.8472582101821899, 2.067650556564331, 1.0725160837173462, 1.4185791015625, -1.4139865636825562, -0.014419088140130043, 0.17930947244167328, 0.8548132181167603, -0.09283076971769333, -0.3416156470775604, 0.5864580869674683, 0.05222664400935173, -0.21705934405326843, -0.5537804365158081, -0.4271925091743469, 1.1231727600097656, -0.36487507820129395, -0.4043198823928833, 0.970199704170227, 0.8374215960502625, -0.27060648798942566, 0.8723494410514832, 0.1507120281457901, -1.143452525138855, -0.8838464617729187, 0.5177234411239624, -0.3421754539012909, -1.9585351943969727, 0.4636874794960022, 1.5297688245773315, 0.8354915380477905, -1.2297650575637817, -0.37860575318336487, -0.8507687449455261, -0.14083895087242126, 0.006596096325665712, 0.6637130379676819, -1.7100856304168701, -0.5388038158416748, -1.3511990308761597, -0.01277826726436615, 1.155657410621643, 1.4414341449737549, 1.2194932699203491, 0.15592065453529358, 1.1123918294906616, 0.24194344878196716, -0.9674756526947021, -1.1569839715957642, -0.8023795485496521, -0.7438960075378418, -0.3211258351802826, 0.08971957117319107, 0.019550757482647896, 0.026383977383375168, -0.10572842508554459, 0.6005841493606567, 0.36143118143081665, -0.9176730513572693, 2.3285467624664307, -0.4213642179965973, 1.1590925455093384, -1.8556294441223145, -0.6757094860076904, -0.7998713254928589, 0.7112873792648315, 0.42958948016166687, -1.9640434980392456, 0.029108194634318352, 1.0683585405349731, -0.5573934316635132, 0.7530953884124756, -1.0162078142166138, -0.33075150847435, -0.8042808771133423, -1.0353225469589233, 0.5253986716270447, -0.61163330078125, -0.08508022874593735, -1.1982091665267944, -1.8569965362548828, -1.6518079042434692, 0.523633599281311, 1.8598271608352661, 1.5232044458389282, -1.277077078819275, -2.0568675994873047, -2.0485894680023193, 1.7267554998397827, -0.658536970615387, 1.9700819253921509, 0.6425184607505798, -0.6991946697235107, 0.4931993782520294, 0.48227211833000183, -0.3290182650089264, 2.330423355102539, -0.5499975085258484, -0.8252546191215515, 0.301791250705719, 0.6783470511436462, -0.4145548641681671, -0.5021944642066956, 0.10174601525068283, 1.252603530883789, 0.4482564926147461, 0.05830337107181549, -0.4746158719062805, 0.41964754462242126, -1.4478693008422852, -1.1802102327346802, 1.1280395984649658, -0.3200586438179016, -0.6606429219245911, 1.570330023765564, -0.4167265295982361, 0.5387173891067505, -0.8292284607887268, -0.22246010601520538, 0.5105292797088623, 1.682944893836975, -1.6959072351455688, 1.192990779876709, 1.577436923980713, 1.8546165227890015, 0.07128400355577469, -0.37454351782798767, 0.2041807472705841, 0.6759412288665771, 2.1125569343566895, 0.17899295687675476, -0.16472667455673218, 0.5571538209915161, 2.448503017425537, 1.5551904439926147, 0.43668603897094727, -0.3200196623802185, -0.44906413555145264, 1.5694968700408936, -0.1676989048719406, -1.1016138792037964, -0.1669401079416275, 0.12178933620452881, -0.1732923835515976, -0.3544592261314392, -1.6565223932266235, 0.26873260736465454, 0.6670977473258972, 1.185925841331482, -0.6995764374732971, 0.5805152654647827, 0.28661859035491943, 0.08714600652456284, -1.1746430397033691, 1.733536958694458, 1.5294204950332642, 0.2992568910121918, 0.9520551562309265, -0.5579448938369751, -0.7078065872192383, 0.40971142053604126, 0.3794083893299103, -0.9847608208656311, 0.9231282472610474, -0.2951475977897644, 1.4300593137741089, 0.6067724227905273, 0.4964108467102051, 0.0005649885861203074, -0.3931054174900055, -0.20100751519203186, 1.4000475406646729, -1.2020187377929688, -1.6521109342575073, 1.1770100593566895, -1.3295613527297974, 0.7316702008247375, 0.04301300644874573, -2.058544635772705, 0.0717083141207695, -0.21456944942474365, 0.41935575008392334, -3.333822250366211, 0.3227628469467163, 0.591036856174469, -0.24872711300849915, 0.8815388083457947, 0.2957915663719177, 1.0435603857040405, 1.0244483947753906, -0.9320314526557922, 0.17101043462753296, 0.6400817036628723, 0.19275304675102234, -1.4450290203094482, -1.4846277236938477, -1.4683480262756348, -1.527543306350708, 2.3341917991638184, 0.48431503772735596, -1.3222734928131104, 0.07144249230623245, 0.27651330828666687, 1.2247799634933472, -1.2212364673614502, 0.460363507270813, -0.9883301854133606, -0.5566835999488831, 0.7553498148918152, -1.9476826190948486, -1.3299424648284912, -1.1291253566741943, -0.2395920753479004, 1.1481332778930664, -1.2947983741760254, 1.0545700788497925, 0.5859386324882507, 0.7916449308395386, 0.5733994245529175, 1.0618680715560913, 0.9860631227493286, 1.9010998010635376, 1.5001561641693115, -0.4548341929912567, -0.06260025501251221, 0.05621423199772835, 1.0088907480239868, 0.9693991541862488, -0.2722715139389038, 0.012148934416472912, -0.8648671507835388, -0.006565313320606947, 0.4408329725265503, -0.07858105003833771, -0.4306851029396057, 0.7950586080551147, -0.5360978245735168, 1.9697177410125732, 0.5349611639976501, 1.40731942653656, -1.501452922821045, -0.4089701175689697, 0.9806561470031738, 0.9074739813804626, -1.160362720489502, 0.6518532633781433, 1.3418916463851929, 0.8422142863273621, -0.6360493302345276, 0.06270834058523178, 0.6504952311515808, -0.6148226261138916, -0.18237292766571045, -1.2791627645492554, 0.14614257216453552, -1.5770257711410522, -0.6124649047851562, -0.48529836535453796, 0.4357199966907501, 0.7374540567398071, 0.25182071328163147, 0.1319350153207779, 0.19168098270893097, 0.34166353940963745, 0.45892518758773804, -1.8387298583984375, -0.05073688551783562, 0.14083220064640045, -0.9694669842720032, 0.20210428535938263, 0.3278181552886963, 1.5268858671188354, -0.31399640440940857, -1.0745453834533691, 0.08744007349014282, -2.2021985054016113, -1.1920678615570068, -1.1000289916992188, 2.2179782390594482, 0.7888999581336975, -1.2451395988464355, 0.19629350304603577, -0.3744703531265259, -0.7962206602096558, -0.27156248688697815, -0.7899160981178284, -1.8614521026611328, 0.7354453206062317, -0.26068809628486633, 1.2442164421081543, -0.7613955736160278, -0.37743839621543884, 1.9104722738265991, 0.1420040726661682, -1.2590678930282593, -0.6662533283233643, -1.2603483200073242, -0.005501341540366411, -0.044373758137226105, 0.6232950091362, -0.6454489231109619, -0.10913657397031784, -0.8399549126625061, -0.7553483247756958, -0.36914771795272827, 0.9034410715103149, -0.3016543388366699, 1.848314881324768, -1.5835635662078857, 0.057812757790088654, -0.04564610868692398, 0.2896878719329834, 0.976436197757721, -0.7528452277183533, -0.5438231825828552, -0.06177219748497009, -2.2358338832855225, -0.3102397322654724, -1.165459394454956, -1.1025047302246094, 1.200658917427063, 0.4367741644382477, 0.03912017494440079, 0.6145390868186951, 0.21362482011318207, 0.5163646936416626, -1.1269896030426025, 1.7714875936508179, 0.6350229978561401, -1.1974469423294067, 0.7127114534378052, 0.3011936843395233, 0.4706130623817444, 0.02691468968987465, -1.3733128309249878, 0.45267191529273987, -0.7042725682258606, -1.3757551908493042, -0.16859644651412964, -2.721832275390625, -0.29171666502952576, 0.45374342799186707, -0.6601651310920715, -0.2202071249485016, -0.9250421524047852, 0.42553654313087463, -0.4835200011730194, -1.1744240522384644, -0.2393164485692978, -1.3763015270233154, 0.12411036342382431, 2.2682580947875977, -1.3759852647781372, -1.2257513999938965, 0.1922137588262558, -0.2782771587371826, -0.7201118469238281, -0.31858178973197937, -0.23009292781352997, -0.7871851921081543, -0.8127090334892273, 1.6898051500320435, 0.21321424841880798, 0.598347544670105, -1.7414718866348267, 0.7726544737815857, 0.9019700288772583, -1.2206025123596191, 0.09073657542467117, 1.2122126817703247, -0.9637466669082642, -0.15310262143611908, 0.5669651627540588, 0.48839619755744934, -0.10570069402456284, -1.8178372383117676, 0.4592444896697998, 1.4450881481170654, 0.6634784936904907, 1.2603060007095337, 1.2414315938949585, 0.9404966831207275, -1.6625086069107056, 0.10386240482330322, 1.185058832168579, 0.10121697187423706, 0.2737284302711487, -2.3122212886810303, -1.103121042251587, -1.6193125247955322, -1.2461341619491577, -1.050550103187561, 2.281827688217163, 1.4926905632019043, 2.413388729095459, 2.0361745357513428, -2.2989556789398193, 0.2742874026298523, -1.0077385902404785, 0.18140564858913422, -1.0257383584976196, 0.17049485445022583, 0.21428342163562775, -0.5556971430778503, 0.7993040084838867, 0.6443098783493042, 1.217010498046875, -1.5563569068908691, -1.3822360038757324, -1.944907784461975, 1.1137608289718628, 0.19121472537517548, 0.30234718322753906, -1.3661731481552124, -0.4306544065475464, 0.9554612636566162, -0.3791203498840332, 0.14560513198375702, -0.2423614114522934, 1.871435523033142, -0.1889084130525589, 0.8196328282356262, 1.6516445875167847, -1.075124979019165, 1.1845437288284302, -0.7558885812759399, 0.882236659526825, -1.0448498725891113, 0.2800390124320984, 0.8038256168365479, 0.3055471181869507, 1.5046684741973877, -0.035975683480501175, 1.0401324033737183, 0.7273856401443481, -1.9390758275985718, -1.4341297149658203, 1.4299157857894897, 0.14686748385429382, -2.931030511856079, -1.964984655380249, 0.006569544319063425, 0.22473658621311188, 2.2758729457855225, 0.181911438703537, 0.3888040781021118, 1.0582672357559204, -1.2015941143035889, -0.1941184103488922, 0.664937436580658, -0.2608087658882141, -0.5477940440177917, -0.02944144420325756, 0.04134276881814003, 1.5215344429016113, -1.0243926048278809, 0.11866315454244614, 0.7145925760269165, -0.13423988223075867, 1.2818890810012817, 0.8286206126213074, -0.8173970580101013, 0.7060449719429016, 0.9269123077392578, 0.030661916360259056, -0.25989705324172974, 0.08707862347364426, -0.6033209562301636, 0.8613618612289429, -0.24548080563545227, -1.7419973611831665, -0.7923887372016907, 0.03258784860372543, -1.0742570161819458, -1.3963557481765747, 1.5299038887023926, 2.167262554168701, 1.6134333610534668, 0.2131495326757431, -1.2409651279449463, 0.5941876173019409, 0.08592548966407776, -1.0383074283599854, 0.9383358359336853, -1.093981146812439, -0.4557208716869354, -2.0009207725524902, 1.1305491924285889, 1.5940563678741455, -0.6518611311912537, 0.20665255188941956, 0.4530718922615051, -1.275864839553833, 1.0353751182556152, 0.2724716365337372, -0.9206640124320984, 1.5426490306854248, 1.7188482284545898, -0.41601672768592834, 0.30847805738449097, 1.3514885902404785, -0.1676466166973114, 0.3625201880931854, -0.5177838802337646, 0.9567868709564209, 1.8784183263778687, 1.0219985246658325, 0.7503214478492737, -0.7054770588874817, -0.21421979367733002, 0.5087954998016357, -0.17068219184875488, 0.9463269710540771, -0.16777026653289795, 1.7561867237091064, -2.4384000301361084, 0.7674070596694946, -1.4491807222366333, 0.20634377002716064, 1.3864113092422485, 0.4721834659576416, -1.3421449661254883, 1.1895681619644165, -0.22229008376598358, -1.0487253665924072, 1.1538784503936768, -0.46198639273643494, 0.6332732439041138, -0.6664882898330688, 0.5732089281082153, 0.42579251527786255, 0.6923578977584839, 0.9635546803474426, 0.1725810468196869, 0.11120206117630005, 0.3743053674697876, 0.5128797888755798, -0.858190655708313, 0.4256991744041443, -0.4618763029575348, -0.8086466193199158, -0.11557216942310333, -0.7090425491333008, 0.2992447018623352, 0.9314107894897461, 1.4539988040924072, 0.38195374608039856, -0.07029858231544495, 1.3629587888717651, -0.46535149216651917, 1.6652472019195557, -0.047415852546691895, -0.9495821595191956, -0.5149421691894531, -1.320603609085083, 0.7461304664611816, -1.5595711469650269, 0.889564573764801, -0.5767688155174255, -0.08824386447668076, -2.0625007152557373, -2.5464460849761963, -0.17162460088729858, -0.2968533933162689, -1.6868271827697754, -0.7694992423057556, -1.3313032388687134, 1.6947506666183472, 0.5254583954811096, -0.33163613080978394, 0.756161630153656, -0.44976651668548584, -0.6385695338249207, 1.5120394229888916, -0.43308159708976746, -0.3436150550842285, -0.5825270414352417, -0.6457198858261108, -2.1482315063476562, -0.3730008602142334, -0.4411310851573944, 0.06140226498246193, -2.3369433879852295, -0.528315544128418, 0.09023570269346237, 0.9647229313850403, 1.2904930114746094, 0.22660300135612488, 0.08581168204545975, 0.9022023677825928, 1.1695386171340942, 0.9418491125106812, -0.5493448972702026, -0.35731571912765503, -0.9342067837715149, 0.7482452988624573, 0.17960646748542786, 0.09894633293151855, 0.24239961802959442, -0.3086274266242981, -0.01983858086168766, 0.5597348809242249, -0.631848931312561, 1.301774263381958, -1.7201967239379883, 0.2765997648239136, 0.1582999974489212, -1.5769925117492676, 0.6093614101409912, -0.03922051936388016, -2.7234511375427246, 0.4223587214946747, -0.12324114888906479, 0.10606769472360611, -0.5976620316505432, -0.07173199951648712, 0.15659388899803162, 0.7650058269500732, 0.394281268119812, 1.1775996685028076, -0.42629313468933105, 2.2481303215026855, -0.07516809552907944, -0.9364465475082397, 0.8703280091285706, 0.5689611434936523, -0.5792673230171204, 1.6103265285491943, 0.646107017993927, -0.6271564364433289, 0.2871832549571991, 1.4035481214523315, 0.4165964722633362, -0.5581463575363159, -0.1560101956129074, -0.24306795001029968, 1.0220694541931152, 2.049718141555786, 1.1987948417663574, -0.37204259634017944, 1.4962304830551147, -1.1194859743118286, 0.31489551067352295, -0.5569654703140259, 0.13712608814239502, 2.074154853820801, 0.7853813767433167, 1.5428719520568848, 0.6937381625175476, 0.14164228737354279, 0.7467255592346191, -0.7436540126800537, -1.5381977558135986, -1.4794633388519287, 1.846057415008545, -0.6872909069061279, 0.3437163233757019, 2.0530905723571777, 0.7420216202735901, 0.42066633701324463, -1.5911370515823364, -0.249114990234375, -1.6206421852111816, -1.0816885232925415, 1.7968806028366089, 0.8869400024414062, 0.1760994791984558, -1.0461945533752441, -1.115118384361267, 1.0226922035217285, 0.23739716410636902, 0.08066494762897491, 0.18837106227874756, 0.6862459778785706, -0.008414564654231071, -1.777641773223877, -0.637778639793396, 1.2933461666107178, 0.2777661681175232, -1.5078577995300293, 0.8339555263519287, 0.17333565652370453, -1.2942265272140503, 0.7538558840751648, -1.585023283958435, -1.338317632675171, 0.8444342017173767, -1.6786398887634277, -0.9496056437492371, -0.36147454380989075, -1.5345540046691895, -0.6520350575447083, 1.226155161857605, 1.862312912940979, 0.21653179824352264, -0.4758748412132263, 0.37616413831710815, -0.8839381337165833, 0.16342878341674805, 0.28394535183906555, -3.0176775455474854, -1.3960438966751099, 0.2800033688545227, -0.8917198777198792, -0.12693661451339722, 0.6688909530639648, -0.3982243239879608, 0.25603345036506653, 0.15692125260829926, 1.3768374919891357, 0.7649869322776794, 0.6904551982879639, 0.4875859320163727, -0.7529711723327637, -2.284019947052002, -0.05201485753059387, -1.3128981590270996, 2.3706698417663574, -2.1545276641845703, 1.713215708732605, -0.4688999056816101, 0.5025488138198853, 1.4634430408477783, -0.8188297748565674, -1.2744436264038086, -0.055714260786771774, 0.17046287655830383, -1.0998573303222656, 0.9636521339416504, -0.06512707471847534, -1.9936665296554565, 0.45202282071113586, -0.012767783366143703, 1.6984641551971436, -0.822184145450592, -0.7840418815612793, -0.009089713916182518, -0.7650521397590637, 0.565687894821167, 0.8865910172462463, 0.5973159074783325, 0.1679709553718567, -0.9539700150489807, 0.915425181388855, -1.4837168455123901, 1.2751773595809937, 0.8223280310630798, 0.7682569622993469, 0.3562586307525635, -0.38085824251174927, 2.337663412094116, 0.454550176858902, 0.6868540048599243, 0.7362661957740784, 0.9556671380996704, -0.24102705717086792, -1.2944787740707397, -0.8587471842765808, 0.46310827136039734, -0.8107220530509949, 0.24294725060462952, 0.9584229588508606, 0.7428323030471802, -0.5110657215118408, -0.9566748738288879, -1.3790240287780762, 0.31370413303375244, -0.49713292717933655, -1.3886520862579346, 1.560745358467102, 2.130570650100708, -2.6880078315734863, -1.2373006343841553, 0.5777714848518372, 0.4717336595058441, 0.8383499383926392, -2.007274627685547, 1.3064501285552979, -0.16208526492118835, 0.006352078635245562, -0.35271739959716797, -0.5048354864120483, -0.35447677969932556, -1.1039186716079712, -1.281807780265808, 0.5784010291099548, 1.2306710481643677, 0.008279270492494106, -0.5897254347801208, 0.30536770820617676, 0.3628388047218323, 1.2630335092544556, 2.460968255996704, 1.5605406761169434, -0.9541450142860413, -2.1097495555877686, 0.35147038102149963, 0.4606628715991974, -0.6479026079177856, 1.4266095161437988, -1.9776393175125122, 0.016412360593676567, -0.7099547386169434, -1.9462283849716187, 0.5025717616081238, -1.6945031881332397, 1.5736262798309326, 1.6305267810821533, 0.6052564978599548, -0.29702436923980713, -0.2363136112689972, 0.26887017488479614, 0.394400417804718, -1.9611432552337646, -0.025793757289648056, -1.1218202114105225, 1.198900580406189, 0.22115261852741241, -0.1093527302145958, -0.8572139739990234, 0.17730781435966492, 1.0894497632980347, 3.078037738800049, -0.9882318377494812, -0.024544408544898033, -0.3882216215133667, -0.02570636011660099, 0.03791866824030876, -0.39222466945648193, -0.026836413890123367, -1.870995044708252, 0.40401116013526917, -0.32341140508651733, 0.18032695353031158, 0.8076798319816589, -1.2390384674072266, -0.08186959475278854, 2.4400339126586914, 1.2197457551956177, 1.372076392173767, -0.08016755431890488, -1.364198923110962, 0.13689686357975006, -1.108046531677246, 0.6935586929321289, 1.1112178564071655, 0.32484158873558044, -0.20099903643131256, -0.2311173975467682, -1.0240141153335571, -1.0144397020339966, -1.4737763404846191, -1.063706874847412, -1.4449331760406494, 0.730516791343689, 0.5663561820983887, 1.5765856504440308, 0.42346349358558655, 0.1354440450668335, -0.12709391117095947, -0.1539757251739502, 1.5233728885650635, 0.3970388174057007, -0.8392472267150879, -0.9928858876228333, 0.5543164014816284, 0.2175980806350708, 0.27792099118232727, 0.6304698586463928, 0.2940514385700226, 0.5039992332458496, 0.35019436478614807, 0.026919761672616005, -0.26307642459869385, 1.327931523323059, 0.22287516295909882, 0.5976842641830444, -1.640863299369812, -0.022691281512379646, -0.41441887617111206, 0.15782839059829712, 0.88614422082901, 0.11456003040075302, -0.8893615007400513, 2.723442316055298, -0.0885041207075119, -1.5415186882019043, 0.8665158748626709, -0.03683819621801376, -0.37966474890708923, 1.0129365921020508, 0.6014150381088257, -0.09883131831884384, 0.49491870403289795, 1.6993467807769775, -0.6300226449966431, 1.2739779949188232, -0.13824255764484406, -0.09980708360671997, 1.5570577383041382, -0.7960089445114136, 0.10698607563972473, -0.5918904542922974, -1.3086010217666626, -0.15351366996765137, 0.4946097731590271, -0.977354884147644, -0.315833181142807, 0.2830753028392792, 0.692357063293457, -0.5064889788627625, -0.6667808890342712, 1.3366435766220093, 1.0666548013687134, -0.5702072381973267, 0.6082044243812561, 0.7947158217430115, 0.6069608926773071, 1.7565317153930664], [0.6390938758850098, 0.23414604365825653, 0.7216914296150208, -1.5105364322662354, 0.8975839018821716, -0.44846004247665405, -1.5569236278533936, 1.0476104021072388, 0.7893529534339905, -1.1515023708343506, -0.7978026270866394, -0.581439733505249, 0.5097237825393677, -0.0796021893620491, 0.1391725093126297, 0.7897566556930542, -0.6348546743392944, -1.1485430002212524, 0.19075623154640198, 0.8542709946632385, 1.1501061916351318, -0.18755057454109192, 0.16009430587291718, 0.2977145314216614, -0.11305403709411621, 0.3501380383968353, 2.4468472003936768, 0.4838654100894928, -1.5461328029632568, 1.8926571607589722, -0.20758922398090363, -0.5905768275260925, 0.580883800983429, -2.3560779094696045, 0.4864039719104767, -0.6576051115989685, -0.5184867978096008, -1.2308393716812134, -0.5942853093147278, 1.6279127597808838, 0.2123146504163742, -0.018321065232157707, -1.796433687210083, -0.23404696583747864, -0.4433947205543518, -0.8426636457443237, 1.715883731842041, -1.0052757263183594, 2.1366724967956543, 1.0666239261627197, 0.6033526062965393, 1.854695200920105, 1.0240222215652466, -0.34647876024246216, -0.21798227727413177, -1.1346787214279175, -1.6889301538467407, 0.5051959156990051, 1.3116241693496704, 0.27775222063064575, -1.4408636093139648, 0.8309170007705688, -0.43372583389282227, -2.836024284362793, -0.14690609276294708, 0.47329428791999817, -1.1782119274139404, 1.1896812915802002, 1.2113683223724365, 0.995223343372345, 0.6418724656105042, -1.4417930841445923, -0.15229664742946625, 0.7610608339309692, -3.4526846408843994, -1.1222350597381592, -1.0422322750091553, -0.5285446643829346, 1.3169561624526978, -0.7718334197998047, -0.310060054063797, -0.266740620136261, 2.7655017375946045, -0.6998350620269775, 0.9019518494606018, -0.9577919244766235, -0.37436965107917786, 0.6738905906677246, -0.5976294875144958, -0.5928283333778381, -0.029483690857887268, -1.2767380475997925, -0.20036636292934418, -0.9929828643798828, 0.7776622772216797, -0.6876973509788513, 0.9919288158416748, -0.6445537209510803, 1.6346122026443481, -1.0397011041641235, -0.8524444103240967, -0.24039527773857117, -0.35957059264183044, 0.3757440745830536, 0.263296902179718, 0.24077549576759338, 1.4124916791915894, 0.8126857876777649, -0.5616282820701599, -0.5025489926338196, -0.6028189659118652, -2.2316079139709473, 2.0691161155700684, 0.10285040736198425, -1.3613171577453613, 0.014742206782102585, -0.06622393429279327, -0.8456861972808838, -1.8300282955169678, -1.3404396772384644, 0.42242303490638733, -0.28895026445388794, -0.5534206628799438, 0.17438700795173645, -0.20294244587421417, -1.2984020709991455, 0.6021761298179626, -0.9131666421890259, -0.32198286056518555, -3.029787540435791, 0.3368566334247589, -0.7751033902168274, -1.3719613552093506, -0.2932763695716858, -0.44708141684532166, -0.36144012212753296, 0.9300358295440674, -0.4078119993209839, -0.04803558066487312, 0.1924518644809723, -0.23127563297748566, 2.084439992904663, -0.9838834404945374, -0.2426328957080841, 0.6943789720535278, 0.10598605871200562, 0.5122342705726624, 1.1284795999526978, -1.05851411819458, 0.7466045618057251, 1.4303187131881714, 1.1720446348190308, 1.39783775806427, 1.2707096338272095, 0.9988691806793213, -0.21845965087413788, -1.3934227228164673, 0.5235975980758667, 0.6597203612327576, 1.00322687625885, -0.2549332082271576, -0.6581036448478699, -1.293237566947937, -0.5119467973709106, -0.21256545186042786, -0.15727555751800537, -1.524710774421692, 1.8607759475708008, 1.7161973714828491, -1.423936128616333, 0.10145916789770126, -0.1813715696334839, -0.04065482318401337, -0.142906054854393, 0.6131637096405029, 0.12524303793907166, -2.3432576656341553, -0.4387945830821991, -0.12330751866102219, 0.22452889382839203, 0.9148320555686951, -2.473560333251953, 0.06100715324282646, 0.35069161653518677, -0.05060138925909996, -1.1013524532318115, 0.4587972164154053, -0.7291350364685059, -0.7040443420410156, -1.6563867330551147, 0.21694597601890564, 1.0607584714889526, -2.7489171028137207, -0.49774280190467834, 0.22988782823085785, 0.10089796781539917, 1.3107190132141113, -1.0627994537353516, 0.2030489295721054, -0.6778820753097534, 0.5817874670028687, 0.28356900811195374, -1.0650384426116943, 0.2602599859237671, -1.0017497539520264, -1.6644179821014404, -0.10414363443851471, 0.3276190459728241, -1.2410767078399658, -0.28416329622268677, 1.7088555097579956, 0.3594929873943329, -1.3144326210021973, -1.5207630395889282, -0.4348367154598236, 1.2690876722335815, -1.0309995412826538, 0.6170623898506165, -2.0001933574676514, 0.15882813930511475, 0.9995378851890564, -1.6984329223632812, -0.6672216057777405, -1.596911907196045, 0.2598065733909607, 0.47798359394073486, -1.0949681997299194, 1.4505759477615356, 1.7623594999313354, 0.008888902142643929, 1.104750156402588, -0.5837762951850891, 0.33479154109954834, -1.9450879096984863, 0.4688783288002014, -1.3530160188674927, 0.07021667063236237, -0.3376205861568451, -0.30670419335365295, -0.4531117379665375, 0.8487510681152344, -1.631988763809204, -1.3551650047302246, 0.2588515281677246, -0.8818650841712952, -0.5528265833854675, -1.4259209632873535, -1.1874885559082031, -1.1509615182876587, -1.6190153360366821, 1.6992250680923462, 0.3511594533920288, -1.4743287563323975, -0.6205236911773682, 1.0176531076431274, 1.0121506452560425, -0.45142802596092224, 0.024016253650188446, 0.7278625965118408, -0.6953417658805847, -0.782757043838501, -1.5208885669708252, 0.9197795987129211, 0.10991220921278, 1.3894261121749878, -0.35964807868003845, 0.8605852723121643, -1.1243815422058105, -1.2330167293548584, 0.12580673396587372, 0.2763356566429138, -1.1418471336364746, 1.269486904144287, 1.1327447891235352, 1.0769010782241821, -1.9749058485031128, -1.6400861740112305, -0.050322238355875015, 0.6803020238876343, -0.1422676295042038, -0.755336344242096, -0.4352419972419739, 0.7572899460792542, -1.7065188884735107, -0.6502907276153564, 2.3886501789093018, -0.06920376420021057, -0.3717767596244812, -0.030983591452240944, 1.7995359897613525, -1.381627082824707, 1.4422866106033325, -0.2669207751750946, -1.4059433937072754, -1.3107367753982544, -0.7341828942298889, 0.36753174662590027, -0.449111670255661, 1.1192506551742554, 2.0961763858795166, 1.4375547170639038, 1.5606005191802979, 1.1152300834655762, 0.6651144027709961, 0.7820066809654236, -1.323222279548645, 1.4409772157669067, -0.26560598611831665, -0.8658139705657959, 0.4485335946083069, -0.43387410044670105, -0.27580830454826355, 0.3086629807949066, -0.5715615153312683, -1.4415392875671387, 0.4322023093700409, -0.6504802703857422, 0.40957963466644287, 1.2984094619750977, -0.7338734865188599, 1.4733656644821167, -0.7433990240097046, -1.0947742462158203, -0.5129053592681885, 1.569453477859497, -1.1716172695159912, 0.08403270691633224, -1.4712637662887573, 1.3011537790298462, -0.841342568397522, 1.4557024240493774, -1.2372851371765137, -0.45427340269088745, -0.40122315287590027, 0.1817144751548767, 0.28731340169906616, -0.5513386726379395, -0.34856587648391724, -1.424094557762146, -0.1527036875486374, -0.5388268828392029, -1.8549455404281616, -1.0766794681549072, -1.5613760948181152, 2.077756643295288, 0.1765420287847519, -0.6871705651283264, -0.8297982811927795, 1.3516815900802612, -0.6094053387641907, -1.4800530672073364, -0.7258409261703491, -0.6156721115112305, 0.03388701751828194, -0.4034479558467865, -1.2597042322158813, -1.0825039148330688, 0.21574266254901886, -0.9990294575691223, -0.5790325403213501, 1.342895746231079, -0.6386675834655762, 0.16850456595420837, -1.425297737121582, -1.6621875762939453, -0.15797878801822662, -0.4029948115348816, -1.0999161005020142, 0.613702118396759, 0.8588820695877075, 1.7912335395812988, 0.6889209747314453, -1.406354308128357, 0.5111554265022278, 0.9440921545028687, -0.6286416053771973, 0.13720521330833435, -0.6090266704559326, -0.5061596035957336, -0.2263609766960144, -0.4898746609687805, 1.9564993381500244, -0.004064193926751614, -0.17084230482578278, -1.0853838920593262, -0.42080172896385193, -0.8929973244667053, 1.231055498123169, 1.440390706062317, 0.027205264195799828, -1.075683355331421, 1.4642466306686401, -1.179924488067627, -1.1212875843048096, 0.7140067219734192, 0.8750743865966797, 1.309611439704895, -1.0439660549163818, 0.42997777462005615, -0.40679222345352173, -1.5170849561691284, -0.061009928584098816, 1.1404790878295898, -0.3494357466697693, 0.056711386889219284, -0.563643753528595, 0.09946821630001068, 0.4033921957015991, 0.9287926554679871, -0.059026673436164856, 0.039480771869421005, 0.16896411776542664, -0.028009358793497086, -0.7719388008117676, -0.3667696714401245, -0.39376699924468994, -1.038388729095459, 0.23138558864593506, 1.1214184761047363, -0.7510807514190674, -0.11825653910636902, 0.4445202648639679, -0.6845725774765015, -0.8619043827056885, -1.0410676002502441, 0.8468539118766785, -0.8450960516929626, 0.9875040054321289, 0.7430766224861145, 1.39595365524292, 0.90290367603302, -1.2860995531082153, -0.9889997243881226, 0.30931615829467773, 0.35480597615242004, 0.1240682452917099, -0.25810641050338745, -1.1526416540145874, -2.073901891708374, 0.20794545114040375, -1.0415983200073242, -0.24447420239448547, -0.0026985681615769863, -0.8489806652069092, -0.3847355246543884, -0.9769598841667175, -0.6055189371109009, -1.8673181533813477, -0.9590492248535156, -0.2433968335390091, 1.1477298736572266, 0.9248397350311279, 1.0581661462783813, 1.4972864389419556, -0.5082871913909912, 0.8414068222045898, -0.3322366774082184, 0.22213095426559448, -0.7304363250732422, -0.6174113154411316, -1.286615014076233, 0.6850902438163757, -0.0717971920967102, 0.6797085404396057, -0.6175835728645325, -0.84935462474823, -0.1688985526561737, -1.068786859512329, 1.6090342998504639, 0.19040881097316742, 0.2758510410785675, 2.9176857471466064, 1.6007896661758423, 0.6705304384231567, -1.574651837348938, -1.0630857944488525, -0.08759346604347229, -0.1637992262840271, -1.5947718620300293, -1.752096176147461, 0.25991854071617126, 0.9878232479095459, 0.20532922446727753, 1.5072743892669678, 0.5405853390693665, -1.0565658807754517, 0.38898488879203796, 1.1402677297592163, 0.1570727378129959, 1.3617007732391357, 0.8287508487701416, 1.1482568979263306, -1.3084248304367065, 0.18403905630111694, -0.9272306561470032, 0.7111921310424805, 0.6585880517959595, -0.35020148754119873, 0.9153002500534058, 2.459827184677124, 1.1061012744903564, 2.1411383152008057, 0.22447232902050018, 0.04700613021850586, 0.01726788841187954, -0.7221946716308594, -0.14714454114437103, 0.05410517007112503, 0.24572500586509705, -0.3269307017326355, 0.08279182016849518, -0.8596555590629578, -1.1015992164611816, -1.7532380819320679, 1.2655199766159058, 1.6761586666107178, 0.2722053527832031, -0.16711752116680145, -1.2830432653427124, -0.8851365447044373, 3.105628252029419, -0.7803317904472351, -0.5411708354949951, -0.8655773997306824, -1.7982922792434692, -0.2323550581932068, -1.0421119928359985, -0.7276957035064697, 1.0940271615982056, 0.18503715097904205, 0.9548236727714539, -0.3455885350704193, 0.468417227268219, -0.42956873774528503, 1.200150489807129, -0.14917859435081482, 0.8604603409767151, 1.0478569269180298, -0.8824452757835388, -0.5416083931922913, -0.7626670598983765, -0.4758196473121643, 1.178341269493103, 0.5225679278373718, -0.2729052007198334, -0.09337840974330902, 1.8925837278366089, -1.6013985872268677, -1.9001564979553223, 0.7557801008224487, -0.7685738801956177, 0.6364532113075256, -0.17643077671527863, 0.6950339674949646, 0.6600375771522522, 1.7822588682174683, 1.453589916229248, -0.0937725380063057, 1.5050618648529053, -0.1766466647386551, -0.6281235814094543, -0.5993674397468567, -0.1833205223083496, -0.10754527151584625, 0.9373048543930054, 0.38351112604141235, -1.8759357929229736, 0.5272701978683472, -0.9851587414741516, -2.3359627723693848, -0.004554713610559702, 2.260341167449951, 0.20418106019496918, -0.6860561966896057, -2.8281290531158447, -1.8776485919952393, 1.3904881477355957, 1.4346449375152588, 2.4534671306610107, -0.104556605219841, -0.49098148941993713, -0.7741607427597046, 0.19513869285583496, 1.1948081254959106, 0.41294628381729126, 0.7305247187614441, 0.43356773257255554, -1.6026725769042969, 0.5092923641204834, -0.882573127746582, 1.5468356609344482, 0.25388264656066895, 1.3482338190078735, -1.224956750869751, 0.7181883454322815, -1.907219648361206, -0.44784650206565857, -0.004509161692112684, 0.6354950666427612, 0.07827518880367279, 0.8190780282020569, -0.3586655259132385, 0.4755244553089142, 1.4833316802978516, 0.09908849000930786, -0.23109778761863708, -0.9145650863647461, -1.0192440748214722, -0.708814263343811, -1.9827300310134888, 0.4832742512226105, -1.4354703426361084, -0.03179404139518738, -2.1000404357910156, 0.5489645004272461, -0.40052345395088196, 0.0011599509743973613, -2.0411312580108643, -0.8537939786911011, 0.8852039575576782, 0.4726143479347229, 0.08794285356998444, -0.7526319622993469, -0.4019692540168762, 0.8419811129570007, 0.2771260142326355, -1.1411395072937012, -2.6259424686431885, -2.0460591316223145, -1.1752769947052002, -1.7391211986541748, -0.46557092666625977, 0.08480780571699142, 0.47380945086479187, 0.8851689100265503, -0.9676412343978882, -0.0646631121635437, -2.633803606033325, -0.4397728741168976, -0.4106610417366028, -0.36934226751327515, 0.7547857165336609, 1.224827766418457, 0.6462958455085754, 0.4161113202571869, -0.3978872001171112, 1.2985962629318237, 0.249183788895607, 0.22129668295383453, -1.2737663984298706, 0.2807711660861969, 0.6932771801948547, -0.5742335319519043, 0.5468510985374451, -0.05527707561850548, 0.3294484317302704, -0.6366990804672241, -0.5230651497840881, -0.5650086402893066, -0.7376587986946106, -0.44661903381347656, -0.9991263747215271, -0.037845999002456665, 0.6293046474456787, 0.2736751437187195, 0.7440537214279175, 1.3820598125457764, 1.0278724431991577, 1.1401140689849854, 1.532161831855774, 0.10391905903816223, -1.7968214750289917, -0.6676787734031677, -1.0886967182159424, 1.9234097003936768, -0.06304950267076492, -0.5623223185539246, 0.3252428472042084, 0.6201802492141724, -0.7174463272094727, -2.673740863800049, 0.3718620836734772, 0.21605262160301208, 1.477407693862915, 0.5183349251747131, -0.6119166612625122, 0.20734886825084686, 1.4750865697860718, 0.16744212806224823, 0.09859704226255417, -0.6107818484306335, 1.0486217737197876, 0.04266778752207756, -0.5154704451560974, -0.2699197232723236, -0.05130893737077713, -1.8358885049819946, 0.4563286006450653, -0.4236166775226593, -0.2044191062450409, 0.8854588866233826, 0.8221457004547119, -1.330890417098999, -0.1884394884109497, 0.7048826813697815, 0.17109304666519165, 0.5263781547546387, 0.23407863080501556, 0.18187950551509857, -0.33279141783714294, -0.22747021913528442, -2.2448184490203857, -0.024223903194069862, -0.5201908349990845, 1.324759602546692, -0.771108865737915, 1.6108061075210571, -0.9624808430671692, -0.41656574606895447, -0.4459226131439209, -0.09937181323766708, 0.23147481679916382, -1.359999656677246, 0.9328712821006775, 0.9013421535491943, -0.7563955783843994, -0.4448834955692291, 1.51656174659729, -1.1006730794906616, 0.4806215167045593, -0.1407487690448761, -0.3792809844017029, -0.7024604678153992, -1.0251882076263428, 0.24125227332115173, -0.14159664511680603, 0.15496674180030823, -1.1720572710037231, -0.7017635107040405, -0.30471158027648926, -0.7096151113510132, 0.618888795375824, 1.1376304626464844, -1.7433165311813354, 0.3820239007472992, -1.6277028322219849, -0.685333788394928, 0.8205558061599731, 0.235804945230484, -0.9707114100456238, -2.4702727794647217, 1.4922500848770142, -1.4218544960021973, -0.9332402944564819, 0.6452776193618774, -1.1035434007644653, 0.17289729416370392, -1.243876576423645, -0.6135310530662537, 1.3154953718185425, 0.8089926838874817, -1.2723236083984375, -1.6386525630950928, 0.4418046772480011, 1.6125432252883911, -0.5149828195571899, 0.5025482773780823, 0.33590343594551086, -3.0706050395965576, 0.6491692662239075, 0.5102179050445557, 1.5121841430664062, -0.48274388909339905, -1.5027718544006348, 0.141960009932518, -0.6143726706504822, -0.03652350977063179, -0.5219358205795288, 0.2566594183444977, -0.8842846751213074, -0.9750653505325317, 0.7371492385864258, 0.12391408532857895, 0.24734175205230713, -1.2522164583206177, -0.48673713207244873, 0.7060822248458862, -1.3250489234924316, -0.02749445103108883, -2.183483123779297, -0.2579168379306793, 0.6161782145500183, 0.18846483528614044, 0.7759567499160767, 1.0764175653457642, 0.42679935693740845, -0.7391241788864136, 0.5297996997833252, -0.07001126557588577, -2.1814510822296143, 0.05657707527279854, 0.9601314663887024, -1.917420506477356, -0.6734195947647095, 0.01716524362564087, -1.1994785070419312, 2.2299249172210693, 0.7833660244941711, -0.09394855052232742, 0.12822796404361725, -0.23278571665287018, 1.3584678173065186, 0.3917842209339142, -0.8951742649078369, 0.08796966075897217, 0.3860602378845215, 0.03870290517807007, -0.6256487369537354, 0.23775839805603027, -0.515940248966217, -0.7332944869995117, -1.3841948509216309, 0.6295815110206604, 1.1098201274871826, -0.8182898759841919, -1.1499868631362915, 0.9316503405570984, 1.5570000410079956, 0.31847381591796875, 0.07612068951129913, -0.701528787612915, -0.522100031375885, -1.0277060270309448, 0.06070682778954506, 0.8581035733222961, -0.9719241857528687, 1.4293279647827148, 0.34273475408554077, -0.3229075074195862, 1.1711031198501587, -0.27766603231430054, -0.9539466500282288, -1.050971508026123, -1.1773762702941895, 0.875156819820404, -0.5002433657646179, 0.21889422833919525, 0.22340524196624756, -1.0388051271438599, 0.27880749106407166, -2.1932737827301025, -0.8270214200019836, 0.3096669018268585, -0.5081263780593872, 0.7367257475852966, -0.8750590085983276, 1.0923353433609009, 0.7249923944473267, 0.6552942991256714, 2.971947193145752, 0.401131272315979, 1.1643900871276855, -0.4806167185306549, 0.7283334136009216, 0.6280872821807861, -0.15895242989063263, -0.5216745734214783, -1.1470694541931152, 1.5682379007339478, 0.6171305179595947, -1.2470353841781616, -1.8094909191131592, 0.946751594543457, -1.6779731512069702, 1.4234710931777954, -1.3136764764785767, -0.5705599784851074, -0.8092229962348938, 1.3056015968322754, -0.25968530774116516, 0.26038843393325806, -1.0739704370498657, -0.011530401185154915, 1.0810275077819824, -1.5618553161621094, 2.3911497592926025, 0.13133282959461212, -1.0361839532852173, 1.3203392028808594, 0.5329751968383789, 0.4718608558177948, -0.24961146712303162, -1.9008409976959229, 0.5147459506988525, 1.9628465175628662, -0.8865817785263062, -0.8467370271682739, -0.7648847699165344, 0.3071414828300476, -1.947700023651123, 0.6165803670883179, -0.02061924710869789, 1.7981057167053223, 1.5869923830032349, -0.43064385652542114, 0.1766904890537262, 0.9962661266326904, 1.4324606657028198, -0.33035385608673096, 0.4492150843143463, 1.3020830154418945, 0.12202978879213333, -0.8518615961074829, -0.7864249348640442, -0.9614267945289612, -0.6055530905723572, -0.6237182021141052, 1.063854694366455, 0.48891353607177734, -0.8450101613998413, -0.5938581824302673, -0.48906514048576355, -1.457371711730957, 0.7659317851066589, 2.1937859058380127, -0.3501618504524231, 0.00995105504989624, 0.843923807144165, -0.27853477001190186, 0.7574810981750488, -1.4070736169815063, 0.8344911932945251, -0.3775402307510376, 1.9782695770263672, 1.232366681098938, -0.8826552033424377, -1.7682713270187378, 0.27229732275009155, 0.23889702558517456, -0.18483784794807434, -2.1851751804351807, -0.27069053053855896, 1.5137099027633667, -1.620187759399414, -1.8243657350540161, 0.19075767695903778, -0.38382548093795776, 0.5888455510139465, -1.2230658531188965, -0.7989871501922607, -1.2156676054000854, 0.5581255555152893, -0.35052698850631714, -1.0911763906478882, 0.21289654076099396, -0.8756276369094849, -0.3796277344226837, -1.1321589946746826, 0.8040096163749695, 0.4322414696216583, 1.3813384771347046, -0.12287231534719467, -1.0361826419830322, -0.5517475605010986, -0.33463054895401, 1.0306241512298584, -1.6510438919067383, 1.2194772958755493, 1.1571074724197388, 1.1816353797912598, -0.9098320007324219, 0.5075670480728149, -1.1416659355163574, -1.1654292345046997, 1.8436411619186401, 2.595353603363037, -1.048810362815857, 1.224129557609558, -0.17695535719394684, 0.08756629377603531, -1.2672054767608643, -1.391109824180603, -0.201182559132576, -0.01575530134141445, 0.3002154231071472, -1.1068577766418457, -0.07521259784698486, -0.27845656871795654, -0.5046767592430115, 0.8891786336898804, 1.1692378520965576, -0.9653825163841248, 0.08940918743610382, -1.043160319328308, 0.9468916058540344, 1.0723778009414673, 0.4490433633327484, 0.19625602662563324, 0.41936707496643066, 0.3387739062309265, 1.292436957359314, 0.9145064353942871, -0.28693607449531555, -1.452745795249939, -0.23190583288669586, 1.757795810699463, 0.46096885204315186, -0.24347254633903503, -1.010396122932434, 0.14685383439064026, 1.1885809898376465, -1.2223420143127441, -2.0421056747436523, -0.08813202381134033, 0.17294219136238098, 0.1470930427312851, -0.3134058117866516, 0.8241657018661499, -0.01808820851147175, -1.0131512880325317, 1.7448850870132446, 0.47279074788093567, 1.0951107740402222, 1.1060179471969604], [0.497310072183609, -1.2339723110198975, 0.6982129812240601, 1.591877818107605, 1.5528184175491333, 0.9774425625801086, 0.17519058287143707, -0.380719929933548, -0.06047682836651802, 1.401739478111267, 0.7346361875534058, -0.45529788732528687, 2.064990282058716, 0.09223747998476028, -0.32782426476478577, 1.5992318391799927, 1.1874490976333618, 1.5709939002990723, -0.7509050369262695, -0.5454280376434326, -0.33966904878616333, -0.26551303267478943, 1.5872373580932617, -0.49613356590270996, -0.9770423173904419, 0.927937388420105, -0.3281625807285309, 0.19970746338367462, -0.4997081160545349, 0.4319281280040741, 0.2687409520149231, -1.763122320175171, -0.07952655851840973, -0.6742739081382751, 0.370683878660202, 1.6686500310897827, -0.26337647438049316, 0.6583070158958435, 1.0133174657821655, 0.7497636675834656, 0.16492648422718048, 0.6839355826377869, -0.6434450149536133, -0.2082832157611847, -1.0010977983474731, 0.40094345808029175, 0.38186997175216675, -1.0464062690734863, 2.167024612426758, -1.5336878299713135, -1.0871723890304565, -0.3343956172466278, 1.6389036178588867, -0.1525225192308426, 2.095390558242798, -0.47947806119918823, -0.8632448315620422, -1.1949925422668457, 0.7474326491355896, 1.104843258857727, -0.3441394865512848, -2.9248154163360596, -1.5019160509109497, 0.9666100144386292, 1.2328455448150635, 0.25320202112197876, -0.2068301886320114, -0.7099280953407288, 1.1844650506973267, 1.068759560585022, -1.4583585262298584, -0.3849906325340271, -2.0108754634857178, 0.24097131192684174, 0.8147942423820496, 2.685473680496216, -0.3040215075016022, -0.5706584453582764, 0.18446190655231476, 0.2738979756832123, -0.7167378067970276, -0.8679417967796326, -0.9050989151000977, -1.8231617212295532, 0.14061953127384186, -1.2529867887496948, 1.7612125873565674, 0.05329052731394768, 0.4259772002696991, 1.7751572132110596, 0.48841699957847595, -0.8168509006500244, 0.8519551753997803, -0.5074450373649597, -1.1006152629852295, -0.7029447555541992, 1.1982336044311523, -0.4625798463821411, -0.3155759871006012, 0.20573283731937408, 1.9448816776275635, 0.03236366808414459, -0.6731614470481873, -0.13838468492031097, 0.38010409474372864, -0.22350643575191498, 2.1746509075164795, 1.0027024745941162, 0.7381043434143066, 0.8281097412109375, -2.3918087482452393, 2.28851580619812, 0.8231897354125977, 0.37980324029922485, -0.6964573264122009, 0.03504689782857895, 0.641934871673584, 1.6258269548416138, -0.6795349717140198, 0.3603259325027466, 1.8584150075912476, -1.4372638463974, -1.3184962272644043, -1.2524243593215942, -0.31666818261146545, -0.8760925531387329, 0.27689602971076965, 0.8618873357772827, 0.6688615083694458, -0.2753058671951294, -1.5112268924713135, -1.2592850923538208, 0.11998377740383148, -0.3384765684604645, -0.001404015114530921, -0.6261043548583984, 2.1550729274749756, -1.6836503744125366, -0.18124502897262573, -2.693798065185547, 1.4118194580078125, -0.12321378290653229, 0.2590665817260742, 1.5895607471466064, -0.08319880068302155, -0.6190317273139954, -0.262724906206131, 1.7528386116027832, -1.1189056634902954, -1.4268380403518677, 0.28866085410118103, 0.08876223117113113, 0.7495023608207703, -0.1769401580095291, -1.4776177406311035, -0.43639469146728516, 1.1058107614517212, 1.2326663732528687, -0.17203417420387268, -0.4139639139175415, 0.04654701426625252, 1.9253219366073608, 0.6964381337165833, 0.3805023729801178, 0.3608782887458801, -1.030979871749878, 1.9880061149597168, -0.25801077485084534, 0.7897303104400635, 0.087681844830513, -0.8767469525337219, 0.7954814434051514, 1.6410048007965088, 0.7781126499176025, 1.3464256525039673, 0.6664767265319824, -0.33851563930511475, 1.0492950677871704, 1.664046287536621, 0.1598530262708664, -2.3804209232330322, 1.2677661180496216, 1.052014946937561, 0.30089688301086426, 1.1456818580627441, -0.5105461478233337, 0.25666481256484985, 0.12244221568107605, -1.047195553779602, 0.10853240638971329, 0.12692785263061523, 1.6048197746276855, 0.7335487604141235, -2.218658447265625, -0.6035524606704712, 0.6109995245933533, -0.6514295339584351, -0.29080167412757874, 0.8603463768959045, -0.1275525689125061, 1.027449131011963, -1.1966516971588135, -0.30181753635406494, 2.1570472717285156, 1.6639440059661865, -0.5620172619819641, 0.1631389707326889, -1.8999528884887695, 0.05602259188890457, 0.2344503253698349, -1.4542975425720215, -0.5527859926223755, -0.40312761068344116, -1.0404568910598755, -1.3178387880325317, -0.38785332441329956, -0.8823736310005188, -0.49808913469314575, 1.4275976419448853, 1.642443299293518, 0.28058910369873047, 0.6434244513511658, 1.220750331878662, 1.7884360551834106, 0.8704164624214172, -0.14151951670646667, 1.113340139389038, 0.6167146563529968, -1.0007083415985107, -0.9120722413063049, 1.11836838722229, 1.229131817817688, 1.614816427230835, 0.05420826002955437, 1.3646721839904785, 1.2602434158325195, 2.3455007076263428, 1.3443775177001953, 0.7746954560279846, 1.006396770477295, 0.22774364054203033, -1.162628173828125, -0.7809052467346191, 1.4025005102157593, 0.4084252119064331, 0.10486076027154922, -0.38077908754348755, -0.40373873710632324, -0.2444724142551422, -0.40434730052948, -0.844325602054596, 0.7680809497833252, 0.18614539504051208, -0.1933601200580597, -0.1507420390844345, 0.39593467116355896, 0.2968330681324005, -0.15608569979667664, -0.8977832198143005, -1.2542948722839355, 0.9385926127433777, 0.1609307825565338, 1.6814748048782349, -0.27946022152900696, 0.6837081909179688, 1.4837002754211426, 1.885457992553711, -1.7969956398010254, -0.7066296935081482, 0.5471499562263489, -0.5586726069450378, -1.4289987087249756, 0.9696221947669983, 1.1503019332885742, -0.003617149079218507, -0.488574743270874, -0.3131311535835266, 0.8448091745376587, -0.19158783555030823, -0.32478049397468567, 1.0008845329284668, 0.506131649017334, 0.21589234471321106, -0.5813826322555542, 0.7424392700195312, 0.963222861289978, 0.746578574180603, 0.24201621115207672, 0.3115634620189667, 0.16600219905376434, -0.7591366767883301, 0.5039486885070801, -0.6307032704353333, 0.30538949370384216, 1.1463922262191772, 0.41840314865112305, -0.4039861857891083, -1.485421895980835, 0.4969020187854767, -0.04808250442147255, -1.6185369491577148, -2.5433199405670166, -0.40109702944755554, 0.3362745940685272, -0.6627915501594543, 2.213456392288208, -0.4405255913734436, 0.6254956722259521, -1.5357023477554321, -0.9211169481277466, -0.03662440553307533, 2.1838369369506836, -0.12273285537958145, -0.6790382862091064, 0.794994592666626, -0.7436904907226562, 0.01985759101808071, -0.44290482997894287, 1.3177136182785034, 0.6375494003295898, -1.4893310070037842, 0.2162541002035141, 0.10527164489030838, -0.20618142187595367, -2.249302625656128, 1.1939958333969116, -1.9323378801345825, -0.6887421607971191, 0.3945907950401306, 0.3792898654937744, -2.0906457901000977, 1.0110799074172974, -0.40594080090522766, 1.5830682516098022, -0.3985941410064697, 1.3489750623703003, 0.17575117945671082, 0.3657183051109314, -0.7313269376754761, -1.824120044708252, -1.989405632019043, -0.15062502026557922, -0.23594807088375092, 1.6186509132385254, 0.2829478681087494, 0.5553147196769714, -1.210732102394104, 0.8386419415473938, 0.37600576877593994, 0.5631402134895325, 0.03950173407793045, 0.24451085925102234, -0.38789525628089905, -0.11963985115289688, 0.19274429976940155, -0.8050166368484497, -0.7889333367347717, 0.14334608614444733, -1.2346922159194946, -1.776813268661499, -0.9119883179664612, 0.5249716639518738, -0.884168803691864, 0.9269172549247742, 0.2213575690984726, -1.8465995788574219, 1.2280054092407227, 0.6688786745071411, -0.2795999050140381, -0.967352569103241, -0.5636931657791138, -0.176332026720047, 0.7336275577545166, 1.3113586902618408, 0.45514169335365295, -0.1385982632637024, 0.4644055962562561, -0.17191210389137268, -1.7448277473449707, 1.0758225917816162, 0.8403002023696899, 0.4034973978996277, -0.8462173938751221, -0.4019198417663574, 1.931076169013977, 1.2001924514770508, -0.012811899185180664, 0.23581336438655853, -3.2224395275115967, -1.9157086610794067, -0.39338356256484985, -0.08821292221546173, -0.41604405641555786, 0.4417584538459778, -0.25032106041908264, 0.12964338064193726, 1.545390248298645, -0.9280867576599121, -0.8291151523590088, -0.6243754625320435, -1.0390785932540894, -0.5727972388267517, -0.48355916142463684, -0.6666799783706665, -2.005357503890991, -0.05510206148028374, -0.1862093210220337, -1.0781950950622559, 0.17002040147781372, 1.2786247730255127, -1.107925534248352, 1.9252928495407104, -0.06430576741695404, -1.3440775871276855, 0.35557281970977783, -1.6273099184036255, 1.8781802654266357, -0.36539387702941895, 0.8201726675033569, -0.8861737251281738, 0.03879867494106293, 0.7308213710784912, -0.6967777609825134, 0.9361668825149536, 0.519694447517395, -0.8194670081138611, -2.393296241760254, 2.2954699993133545, 1.5865827798843384, 1.752376914024353, -1.5121750831604004, -0.05264643579721451, 0.5113075375556946, 0.1192086935043335, -0.3845347762107849, 0.5891337394714355, -0.30956169962882996, 0.6404399275779724, 1.550005555152893, 0.8597492575645447, -0.4055655002593994, -1.546981930732727, 1.1343194246292114, 0.7656811475753784, -1.1125479936599731, -0.9586836099624634, -0.345659077167511, 1.2552589178085327, -0.10430184006690979, -0.17129407823085785, -0.826678991317749, 0.05602695420384407, -0.762954831123352, -1.4630368947982788, -1.7876557111740112, -0.6439782977104187, -0.9920938014984131, 0.5041865110397339, -0.36900532245635986, 0.8283224701881409, -0.1555870771408081, 0.3514226973056793, -1.0685943365097046, 0.12294376641511917, -2.061650514602661, 0.8349387049674988, 0.6158546209335327, -0.6202178597450256, 0.4169659912586212, 0.5627673864364624, -1.564152479171753, -0.14013460278511047, -0.8465588688850403, 0.4272020757198334, 0.14125995337963104, 0.3012009859085083, -0.4561087489128113, 0.011168459430336952, -0.9233005046844482, -0.9225634932518005, 1.051780104637146, 1.1407577991485596, -0.5274434685707092, 0.12376411259174347, -0.8456433415412903, 0.32844972610473633, -0.4553012549877167, 0.3163074851036072, 0.32960933446884155, 0.8941716551780701, 1.0301662683486938, -1.0452911853790283, 0.6883025169372559, -0.7021707892417908, 0.4751051962375641, 0.5545995235443115, 1.5782973766326904, 0.6489911675453186, -0.28399646282196045, 0.0443711057305336, 0.7198501229286194, 0.08017762005329132, -1.0167051553726196, 0.5485000014305115, 0.2325887829065323, -1.690958023071289, 0.3265745937824249, -0.703478991985321, -0.683721661567688, 0.5727890729904175, 0.6786258816719055, 1.5240106582641602, 0.1540410816669464, -0.1940043419599533, 0.14283233880996704, 0.7368339896202087, 0.0511263832449913, -1.48012375831604, -0.37241330742836, 1.7267431020736694, 0.28019121289253235, -0.32347774505615234, 1.0262932777404785, 0.1631065309047699, 0.48317405581474304, -0.28395774960517883, 0.9243322014808655, -1.0255144834518433, 1.850285291671753, 1.978102684020996, -1.6097687482833862, -0.2216850370168686, 0.8782409429550171, 0.2885679602622986, -0.2608029544353485, 0.352265864610672, 0.27379047870635986, 2.2878525257110596, 0.16555805504322052, 0.08281441032886505, -0.25333496928215027, 0.795676052570343, -1.7423057556152344, 1.1626113653182983, 1.263136863708496, -0.04738571122288704, 1.8985304832458496, 2.4963462352752686, -1.2320197820663452, 2.115130662918091, 0.014375574886798859, 0.6967690587043762, 0.6556971073150635, 0.3502410352230072, -0.35663607716560364, -1.1932477951049805, 0.4208758473396301, 2.066324234008789, 0.5402974486351013, -0.8510244488716125, 0.6812003254890442, 0.7515243291854858, 1.666091799736023, 1.1145304441452026, -0.05558120459318161, 0.8059970736503601, -1.2439465522766113, -0.2680202126502991, 0.7595488429069519, 0.5683025121688843, -1.039067268371582, -0.3950762450695038, -0.8734883069992065, 0.28437289595603943, 0.07779049873352051, 0.731103241443634, -2.192880868911743, -1.3869547843933105, 0.6428155899047852, -0.3860794007778168, 0.7061670422554016, -1.1289317607879639, -0.38849642872810364, -0.7027594447135925, 0.277098149061203, -0.5658591389656067, 0.3294520080089569, 2.0586495399475098, -0.03575439378619194, 0.05414906516671181, 0.6909842491149902, 0.16477349400520325, 1.4174009561538696, -0.87770676612854, 0.4441111385822296, 0.08698412030935287, -0.5989622473716736, 0.26796555519104004, 2.637951135635376, 0.6130149364471436, -0.45034876465797424, 0.21412880718708038, -0.6344495415687561, -1.2515475749969482, 1.1032947301864624, -0.7025070786476135, 1.2073848247528076, 0.4986155927181244, 0.083483025431633, 1.755068302154541, 1.015819787979126, 1.029951810836792, -1.8384177684783936, 0.9494843482971191, 1.600660800933838, -0.6467449069023132, 0.12073943763971329, -0.919574499130249, -1.3902767896652222, -0.87419593334198, -0.3896764814853668, -0.02839401178061962, 0.5818188786506653, 0.9428443312644958, 0.1996368169784546, 1.1784558296203613, 0.4508044421672821, 0.7402141690254211, -0.12722773849964142, 1.2112032175064087, -0.345087468624115, 0.5053313374519348, -0.5177600979804993, 1.0491880178451538, 0.4099932014942169, -0.019667861983180046, -0.34561216831207275, -1.0457507371902466, -1.31069815158844, 0.5904034972190857, 0.38948383927345276, 0.25850364565849304, -1.050422191619873, 1.5581235885620117, 0.8284572958946228, -0.7469018697738647, 0.7276543378829956, 0.41228336095809937, 0.03530534729361534, -1.3779603242874146, 0.7313565015792847, -0.14957267045974731, 0.07203105837106705, -0.11164136230945587, -1.5815538167953491, -0.3024296164512634, 0.3121136426925659, -1.6714540719985962, 0.33713293075561523, -0.17338860034942627, -1.4587736129760742, -0.2604576051235199, -0.7765009999275208, 0.4164050817489624, 0.8301425576210022, 0.7173647880554199, -1.6047366857528687, 1.5610144138336182, 0.1317589282989502, -0.8431178331375122, -0.020767083391547203, 0.7405868172645569, 0.08814385533332825, 0.18171700835227966, 1.168209195137024, -0.7326568365097046, 1.3192813396453857, 0.4358026385307312, -0.40204039216041565, 0.007577556651085615, 0.3472811281681061, -1.2562360763549805, 1.0939847230911255, -1.70121431350708, 1.2411692142486572, -0.8692420125007629, -0.07276221364736557, -1.4472925662994385, 0.6509652733802795, 0.45517316460609436, 0.14983367919921875, -0.3949771523475647, 0.8777525424957275, -1.292873501777649, -1.0590829849243164, -0.7262331247329712, 0.8383211493492126, 0.8318485617637634, 0.6763779520988464, 0.11429668962955475, 1.4712648391723633, -0.6602091193199158, -1.4066144227981567, -1.2955024242401123, -0.7667511701583862, 1.3056327104568481, -1.00465726852417, 1.7163372039794922, -0.8124809861183167, -0.5108264088630676, -0.11454872041940689, -1.0409718751907349, 0.6348854303359985, -1.9201804399490356, 1.0999305248260498, -1.6210047006607056, -1.0099749565124512, -0.6118938326835632, 1.1300420761108398, 0.5927320718765259, -0.23103252053260803, -0.7982763051986694, 0.09970597177743912, 1.629274845123291, 0.28711915016174316, 0.28936994075775146, 0.7918829321861267, 0.20916399359703064, -0.10279387980699539, 0.2223227471113205, -0.6462877988815308, -0.11283041536808014, 0.7224450707435608, -1.5083262920379639, -1.029550313949585, -0.44673168659210205, 1.514732837677002, -0.6939635872840881, -0.6234039068222046, 1.0635312795639038, 0.10474881529808044, -0.9143599271774292, -0.22035318613052368, -1.5012258291244507, 1.1970218420028687, -0.602414608001709, 1.089741826057434, 0.4700310230255127, -0.25224462151527405, -2.0414743423461914, -0.8426634669303894, -0.9727705717086792, 0.45148688554763794, -0.08221826702356339, -0.9526214003562927, 0.7532545924186707, 0.49736693501472473, -0.9184812307357788, -0.12209994345903397, 0.7990075349807739, -0.2827686071395874, -0.18926167488098145, 0.32494592666625977, 0.4690333604812622, -1.925739049911499, 0.8932210206985474, 0.5598952174186707, 0.03488583490252495, -0.5937706232070923, -0.7499615550041199, -0.20146265625953674, -0.1424175500869751, -0.03750375285744667, 0.9228028655052185, 0.8055717945098877, -1.5915699005126953, -0.9594089984893799, 0.8419767022132874, 0.19393177330493927, 0.7365483641624451, 1.9687331914901733, -0.2380572259426117, -1.2657318115234375, 0.8805492520332336, -1.5809910297393799, -0.49506187438964844, -0.6128204464912415, -0.4745458960533142, -1.1644785404205322, -0.5493069291114807, -0.9104666709899902, -1.7756268978118896, -0.8443498015403748, 0.6395635008811951, 0.14922963082790375, 1.1009992361068726, 0.21589654684066772, 1.089489459991455, -1.0207957029342651, -0.7658630609512329, 1.5902838706970215, 0.5391676425933838, 0.8621593713760376, -0.054542120546102524, 0.9489275813102722, -0.7277330756187439, 1.31167733669281, 0.9827284812927246, -0.976192057132721, -0.5738918781280518, 0.7877612709999084, -0.6072533130645752, -0.44947922229766846, -1.017212152481079, -0.6553928852081299, 1.1492946147918701, -1.68485689163208, -0.3613578975200653, 1.0573657751083374, 0.4058198034763336, 0.2691543400287628, 0.47612378001213074, 0.8151960372924805, -1.892491102218628, -0.01800576038658619, -0.29306721687316895, -1.0327316522598267, -0.5323805809020996, -0.06424592435359955, -1.1015136241912842, 0.1466221809387207, -1.1363990306854248, 0.4106642007827759, 0.5400710105895996, 0.1916927546262741, 0.598818838596344, -2.339749574661255, 1.9890227317810059, -0.9664458632469177, -1.160987377166748, -0.1376117467880249, 1.272540807723999, 0.07144171744585037, 0.6209387183189392, -0.4023604691028595, 1.58479642868042, 1.470278024673462, -0.5295997858047485, -0.6953456401824951, 1.1255921125411987, -0.49384260177612305, -0.29445192217826843, 0.6483427882194519, -0.10266848653554916, 0.5634724497795105, 0.42472559213638306, 0.04562525078654289, -0.7828474640846252, 0.5877240896224976, 0.5198884606361389, -0.6877084374427795, -0.29096630215644836, -0.35591262578964233, -0.23104582726955414, -1.0489826202392578, 0.024530580267310143, 0.0587509460747242, 1.3033878803253174, 0.4768383800983429, 0.6159889101982117, 0.11187797039747238, -1.041572093963623, 0.06226332485675812, -1.8279865980148315, -0.6182624697685242, -0.6200124025344849, -0.14861992001533508, -0.11441388726234436, -1.773879885673523, -1.3462481498718262, 0.5650973320007324, -1.336350917816162, -0.6530818343162537, 1.049718976020813, -0.6887463927268982, 0.5101129412651062, 0.8261473178863525, -1.9366345405578613, 0.8812280893325806, 0.6147264838218689, 0.38512149453163147, -0.8939676880836487, 0.4230654537677765, -0.7588814496994019, 0.18248164653778076, 1.0956159830093384, -0.6975052952766418, 1.5612549781799316, 0.910240113735199, 0.5348669290542603, -1.7638965845108032, 0.8624626994132996, 1.900114893913269, 0.9709025621414185, 0.5188425183296204, -0.09792827069759369, -0.9159258604049683, -0.575140118598938, -0.6886938214302063, 0.07374505698680878, -1.792734146118164, -1.9064596891403198, -0.6860304474830627, 0.06404364109039307, 0.5376589298248291, -0.8407713770866394, 0.4612584114074707, -0.3295208513736725, 0.19895370304584503, 0.17333227396011353, 1.1678752899169922, 0.7015812993049622, 2.19301700592041, 0.7190685868263245, 2.146623373031616, -1.647286295890808, -1.8530428409576416, -0.16402332484722137, 0.25823166966438293, -1.0250712633132935, -0.29773882031440735, -0.024174947291612625, 0.5392097234725952, -0.008832630701363087, -0.40521112084388733, -1.3426231145858765, 0.9067198038101196, 1.1011309623718262, 0.7347292304039001, 2.4061036109924316, 0.30692315101623535, -1.0855600833892822, 0.1527954488992691, -1.6315981149673462, 0.43745213747024536, 0.6801607012748718, -0.02135361358523369, 2.1745777130126953, -2.420731544494629, 0.712452232837677, -0.42558416724205017, 1.3839772939682007, -0.38518428802490234, -0.5409021377563477, 0.7480231523513794, 1.9368085861206055, -0.4561622738838196, 0.9433964490890503, 0.11088575422763824, -0.11254903674125671, -1.066311001777649, 0.808673083782196, -0.7546863555908203, 0.28400641679763794, -0.7138871550559998, -1.4599424600601196, 0.408296138048172, -0.3410370349884033, -0.5475521087646484, 0.7987686395645142, 0.28379034996032715, -0.9566163420677185, -0.8781887292861938, 1.9699268341064453, -1.1728649139404297, 0.9670925140380859, -0.014708302915096283, 1.6901705265045166, 0.45417147874832153, -0.9144262671470642, -0.14048294723033905, -0.7930169105529785, -0.12533798813819885, -0.5595957636833191, 1.4126245975494385, 0.2004329413175583, 0.1310712695121765, -0.47272446751594543, -2.0265092849731445, 0.39624106884002686, 1.5811539888381958, -0.7998440861701965, 1.0800617933273315, -1.707646369934082, -0.48933228850364685, 1.722826600074768, 0.06904299557209015, 0.3690761923789978, -1.6939135789871216, 1.5221600532531738, 1.7244856357574463, 0.16602273285388947, 0.6067512631416321, -1.597991704940796, 1.465922474861145, -0.8672683238983154, 0.7945584654808044, 1.3726271390914917, 0.8038339018821716, -0.52872234582901, 0.16491703689098358, -1.2584497928619385, -0.19532237946987152, -0.03935426473617554, 1.115181803703308, -0.7451403737068176, -0.6953160166740417, -1.0908453464508057, 0.9994144439697266, -0.920407772064209, 0.10530470311641693, -0.017957089468836784, -0.019748494029045105, 0.7648733258247375], [-0.8315317630767822, 1.8208781480789185, -1.3793845176696777, -0.5685566663742065, 0.8791682720184326, -2.1505815982818604, -0.4393737018108368, 1.1629300117492676, 0.3584596514701843, 1.6701289415359497, -0.06755083054304123, 1.4168894290924072, 0.2587061822414398, -1.0770286321640015, 0.6188660264015198, -0.15357491374015808, -0.4148785471916199, -0.37498006224632263, -2.6552751064300537, 0.00764030497521162, -0.6475696563720703, -0.7539355754852295, -0.9614241123199463, 1.001693844795227, -0.38622018694877625, -0.18603835999965668, 0.617913544178009, 0.2777915596961975, 0.33358460664749146, 0.2918611466884613, 0.46158698201179504, -0.37085816264152527, -0.8624143004417419, 1.895567536354065, 1.2939003705978394, 1.5904756784439087, 1.08577561378479, 0.6707611680030823, -2.2511844635009766, -0.8482547402381897, 0.983802080154419, 0.8951308727264404, -2.37125563621521, 0.5733707547187805, -0.9730584621429443, -1.333459496498108, -0.20200495421886444, -0.36687344312667847, -0.2838971018791199, -0.20760242640972137, -0.5811036825180054, -0.3308161795139313, -0.2745324671268463, 1.02238130569458, 0.2552012801170349, -0.4283488690853119, 1.8752185106277466, 1.2097845077514648, -1.1403404474258423, 0.3431730568408966, 0.1584392488002777, 1.261326551437378, 0.395652711391449, 0.9706052541732788, 1.4495857954025269, -1.1886484622955322, 0.6887075304985046, -2.099444627761841, 0.12891605496406555, 0.7357022166252136, 2.201869249343872, -0.53118497133255, -0.3752632737159729, 0.45595279335975647, -0.5521910190582275, -0.8697754740715027, -0.5738324522972107, -0.500998318195343, 0.11698079854249954, 0.8564884662628174, 0.6358364820480347, 0.5434383749961853, 1.1186274290084839, 1.1696768999099731, 0.7411330342292786, -0.704800546169281, 0.5525232553482056, -0.13712401688098907, -0.059757258743047714, 0.593726634979248, -1.530275821685791, 2.2026922702789307, 0.02683655358850956, -0.2508354187011719, -1.628350853919983, -1.3591030836105347, 0.05394444987177849, 0.38052189350128174, 1.1450080871582031, -1.673123836517334, 0.5827889442443848, -1.5270185470581055, -0.4228053390979767, -1.3297631740570068, -0.34793567657470703, -0.4069310426712036, -0.9261295795440674, -0.3227921724319458, 0.5764612555503845, -1.4479432106018066, -2.1006827354431152, 0.5244009494781494, -0.14051619172096252, -1.5294073820114136, 1.185107946395874, -1.40296471118927, 0.2420915961265564, 2.266230583190918, 0.9190876483917236, -1.077618956565857, 1.5797799825668335, 0.7218197584152222, -0.2663896381855011, 0.8125666379928589, -0.5286626219749451, -0.9887104630470276, 0.45922738313674927, -0.5392274856567383, -1.5077933073043823, 0.23103614151477814, 2.330310821533203, 0.18493664264678955, 0.16557708382606506, -0.5344317555427551, -0.3152925372123718, -0.6903210878372192, 0.8969528079032898, -0.12679973244667053, -0.07251988351345062, -2.9221553802490234, 0.9151772856712341, 0.7050650119781494, -0.48115643858909607, 0.2320806384086609, 0.34245845675468445, -0.24867913126945496, 0.4063892960548401, 0.8592172265052795, -1.022186279296875, 1.8137333393096924, 0.7619394063949585, -0.49364590644836426, -1.06692373752594, 0.30924564599990845, -0.31751179695129395, 0.09556698054075241, 0.5579593181610107, 1.670964241027832, -0.8657219409942627, -0.5202994346618652, -0.00468854233622551, 0.5603930950164795, 1.3450084924697876, 0.834782063961029, -0.14972545206546783, 1.2319128513336182, 0.8173351287841797, -0.568269670009613, -0.2528432011604309, -1.4186745882034302, 0.5419889092445374, -1.8433823585510254, 0.052104637026786804, 0.29156872630119324, -0.680117130279541, 0.10699032992124557, 0.7808703780174255, 1.2347968816757202, 0.6454781889915466, -0.5676946043968201, -1.1526731252670288, -0.45009398460388184, -0.798421323299408, 0.7254515290260315, 1.3038759231567383, -0.06384363770484924, -0.1618211567401886, -0.686154842376709, 0.42739203572273254, 0.3418196737766266, -0.7964217066764832, -0.5757145285606384, 1.4014949798583984, 1.4159871339797974, -0.25295835733413696, 0.9125812649726868, -1.2866917848587036, 0.2223094403743744, -1.0114903450012207, -0.9978805184364319, -0.5239149332046509, 0.07291882485151291, 1.0739052295684814, 0.7677366137504578, -0.24132947623729706, 1.490622639656067, 0.3539039194583893, 0.3229033946990967, 1.0248703956604004, 0.2572624683380127, 0.24660994112491608, 1.530442476272583, -0.5118188858032227, 1.1112377643585205, -0.5062074661254883, 1.032662272453308, -0.8728002905845642, 0.2701444625854492, -1.0622644424438477, 0.18171177804470062, -2.0338122844696045, -0.39214420318603516, 0.5172119736671448, -0.896496057510376, -0.7561578750610352, -1.3853987455368042, -0.3151942193508148, 1.0549553632736206, -0.7821274399757385, 0.3973616063594818, 0.23860500752925873, 0.03134087100625038, -0.2611311376094818, -0.73700350522995, -0.9267004132270813, -0.14826954901218414, -0.29799482226371765, -0.6585811376571655, -0.9794198274612427, -0.688156247138977, 0.6925415396690369, -1.7390222549438477, -0.8923417329788208, 1.0568559169769287, -1.1367493867874146, -0.7407709360122681, 0.16835416853427887, -0.8122860789299011, 0.2722966969013214, 0.32215410470962524, 1.7503998279571533, 0.5457825064659119, -2.149705171585083, 2.1923930644989014, -0.693107545375824, 0.7676950097084045, -0.9038862586021423, 1.6500128507614136, -0.11161169409751892, 0.6935665011405945, 1.3718898296356201, -0.09213320910930634, -0.47225221991539, -1.081336259841919, 1.9237979650497437, 0.6015616059303284, -1.55574369430542, -0.4511549770832062, 0.3718811571598053, -0.9111201167106628, 0.6601576805114746, -1.1544764041900635, 0.9179569482803345, 0.23634228110313416, -0.4889613687992096, 1.368648648262024, -0.4886179566383362, 1.1832776069641113, 2.1388514041900635, 0.026620622724294662, 1.750777244567871, -2.0653774738311768, 0.5241424441337585, 2.0148377418518066, 0.3193150460720062, 1.8267765045166016, 0.1578138917684555, -0.6612604260444641, -0.8026195764541626, 1.1896575689315796, -0.5171412825584412, 0.30823177099227905, -0.4309389591217041, -0.36527594923973083, -1.130183458328247, 0.9629618525505066, -1.3212357759475708, 1.6784430742263794, 0.7611140608787537, 1.82565176486969, 0.1617051512002945, 0.23142316937446594, 0.29490166902542114, 0.3251168429851532, 0.2603982090950012, -0.3380790948867798, -0.6539612412452698, 1.6874903440475464, -0.9670277833938599, -1.3258485794067383, 0.6009973287582397, 0.1679142415523529, -0.6807448863983154, 2.4359822273254395, -1.855306625366211, 1.2494173049926758, 0.42675504088401794, 0.5135474801063538, 0.5228466987609863, -0.9711813926696777, 1.1739016771316528, -1.5268787145614624, -0.903157651424408, 2.185479164123535, -0.5463663339614868, 0.13281826674938202, 1.6319260597229004, 0.0059363762848079205, 0.5162991285324097, -1.7169831991195679, 0.684122622013092, 1.4873859882354736, 0.6118466258049011, -0.2975998520851135, -0.45476803183555603, -1.0336178541183472, 0.037786319851875305, 0.6415318250656128, 1.2793278694152832, -0.6512476801872253, 0.32525137066841125, 0.9426422715187073, -0.7664355039596558, 0.7862589955329895, 0.1688081920146942, -0.950190007686615, 1.076906681060791, 2.226202964782715, 0.9453168511390686, 0.28491726517677307, -0.09605591744184494, -2.722515344619751, -0.19870680570602417, 1.228068470954895, 1.55146062374115, 0.905436098575592, 0.20479269325733185, -0.7633318901062012, -2.359578847885132, 1.2495698928833008, -1.0678668022155762, -0.042659059166908264, -0.7794370651245117, 0.8150878548622131, 0.16340434551239014, -0.26840445399284363, -0.5067272782325745, 0.22583690285682678, 2.1954727172851562, 0.09208142757415771, 1.0124680995941162, 0.5178959965705872, -0.1206049844622612, -1.4867584705352783, 1.8700720071792603, 0.008129756897687912, -3.184544563293457, -0.52762770652771, -0.6973056197166443, 1.6650482416152954, -0.6909570693969727, -0.35772639513015747, 1.3440316915512085, 0.07732656598091125, 1.496641993522644, 1.0572943687438965, 0.647373378276825, 0.06864012032747269, 0.014118191786110401, 1.6145901679992676, 0.09731549024581909, 0.3882255554199219, -0.004962587263435125, -0.7412134408950806, 0.5341077446937561, -1.335089087486267, 0.17071455717086792, -0.8603240251541138, 0.3146207630634308, -1.4746557474136353, -1.4782359600067139, -2.2120602130889893, -0.8415156602859497, 1.2661746740341187, 0.22407276928424835, -0.3970703184604645, 1.748435616493225, -0.5034992694854736, 1.0932590961456299, -0.790809690952301, 1.3761425018310547, 1.2695711851119995, -0.8424913883209229, 1.558324933052063, -0.3515242040157318, -0.8414413928985596, -1.5766650438308716, -0.7099215984344482, -0.5758270621299744, -0.3981471657752991, -0.29081642627716064, -0.0651598796248436, 0.42387062311172485, 0.39783957600593567, 0.6821995377540588, 0.669766902923584, 1.2213457822799683, -0.5384377837181091, -1.0270566940307617, 0.3943730890750885, -0.07165642082691193, -0.6634575724601746, 1.1990203857421875, -0.11573266237974167, 0.7734248638153076, 0.48398080468177795, -0.6439622640609741, 1.4841079711914062, -0.12617887556552887, -0.9708899855613708, 0.9612246155738831, -0.24324330687522888, -0.4315936267375946, 0.26897403597831726, 0.24819816648960114, -0.857078492641449, 0.678858757019043, 0.05682896450161934, -2.1541824340820312, -1.421091079711914, -0.7158372402191162, 1.308467984199524, 0.7343975901603699, 0.14613644778728485, -0.5923055410385132, -0.4032994210720062, 0.43695589900016785, 0.24147556722164154, -0.07851232588291168, -2.282130718231201, -0.5848581194877625, -2.7027244567871094, -0.013621272519230843, -0.5799430012702942, -0.7171439528465271, -1.8546547889709473, -0.08742856234312057, -0.8186479806900024, 1.0034937858581543, -1.2750822305679321, -1.2672590017318726, 0.15670667588710785, 0.24981452524662018, 0.04441176354885101, 0.8611871004104614, -0.8124621510505676, -0.5851238369941711, 0.3761681318283081, -2.4210410118103027, 0.12177284806966782, 1.1787840127944946, 0.1806030124425888, -0.6703141331672668, 2.490696668624878, 0.8443396091461182, 0.9352870583534241, -1.7021222114562988, -2.0169661045074463, -0.20655333995819092, -0.8699463605880737, -0.9865783452987671, 0.36123186349868774, 1.821602702140808, -1.0316252708435059, -0.8399326205253601, 0.6344358921051025, -1.368820309638977, 0.3579671382904053, 3.0956714153289795, 1.2036250829696655, -0.32111266255378723, -0.33882811665534973, 1.854256272315979, -1.0787931680679321, 0.4275582432746887, 0.9844974875450134, -0.7310723066329956, 0.867047905921936, 1.2560577392578125, -0.9027894735336304, -0.6627349853515625, 0.6032410860061646, -3.2824349403381348, -1.5611602067947388, 1.9480979442596436, 0.4298759996891022, 0.8115668296813965, -0.2287137508392334, -1.5361658334732056, -0.43200305104255676, -0.9593332409858704, -0.7090843915939331, -0.8233035802841187, -0.1773216277360916, -0.736213743686676, 1.1509383916854858, -0.7169284224510193, 0.6023942828178406, -0.4740029275417328, -0.8483033776283264, 0.09188622236251831, 1.3094334602355957, -0.21920889616012573, 0.05882776901125908, -0.2806819975376129, -0.08913916349411011, 0.5512661933898926, -1.4791929721832275, 0.4925440549850464, -1.1300371885299683, -0.31321433186531067, -0.9759162664413452, -0.23260492086410522, 2.2775650024414062, -0.12360386550426483, 0.2841559648513794, -1.8384406566619873, 1.0023953914642334, -0.6312269568443298, 0.19888998568058014, -0.9760188460350037, -1.6434329748153687, 0.21699975430965424, 1.7832894325256348, -0.5189231634140015, -1.4708341360092163, 1.2142513990402222, 0.5467202067375183, -1.78662109375, -0.7815580368041992, 0.5173674821853638, 2.2738125324249268, 1.5633633136749268, -0.16501612961292267, -0.510949432849884, 0.849229633808136, 1.8883635997772217, -1.445175290107727, -0.3097118139266968, 2.003901481628418, 1.092592477798462, -0.5311623215675354, 0.6183158755302429, 0.31076472997665405, -1.0423734188079834, -2.878059148788452, -0.011832193471491337, 0.005808970425277948, 0.869106650352478, -0.8205675482749939, -1.395811915397644, 0.24338170886039734, 1.741119384765625, 0.5115264058113098, 0.011603973805904388, -0.8175961375236511, -0.25738731026649475, 1.1329134702682495, 0.588763415813446, -0.0682336762547493, 0.6331800818443298, -0.5037044882774353, -1.1485408544540405, -0.69953453540802, 1.4542354345321655, -0.09535841643810272, 0.3291130065917969, 0.5779677629470825, 0.33038589358329773, 0.8915634155273438, -0.6118886470794678, 0.15046271681785583, 0.7008315920829773, -1.3763052225112915, -1.715159296989441, -0.5716742873191833, 0.5362938046455383, 0.16725586354732513, -0.24755512177944183, 0.33781859278678894, 0.5651965737342834, 0.49242472648620605, -0.42423015832901, 1.333335041999817, 1.5027674436569214, 2.081171751022339, 0.5001853108406067, 0.2907685935497284, 1.596860408782959, 0.817436158657074, -0.23153045773506165, 1.9281904697418213, -0.90146404504776, 0.9456092119216919, -0.9093058705329895, 0.567035973072052, -0.33725202083587646, 0.7357417941093445, -0.5054487586021423, -0.245831698179245, 0.31040307879447937, -1.7931081056594849, -3.1882996559143066, 1.586650013923645, 0.44341665506362915, -0.2919115722179413, 0.6195464730262756, -0.7696415781974792, 0.15795737504959106, 1.216925024986267, -1.1314281225204468, -0.5955973267555237, -0.10822702944278717, 0.5547772645950317, 0.6156949996948242, 0.8217950463294983, 0.25356513261795044, -1.9916915893554688, -0.02862086333334446, 0.8221880197525024, 0.1467324048280716, -1.8696845769882202, 0.3306564688682556, -1.1651180982589722, 1.0406793355941772, 0.05401349812746048, -1.081113576889038, -1.9739975929260254, -0.48857855796813965, 0.13839152455329895, 0.3086608052253723, 2.2076573371887207, -1.4656031131744385, 0.9726758003234863, -0.4863150119781494, 0.3435516655445099, 1.7411000728607178, -1.4596171379089355, 0.9950054287910461, 2.1016836166381836, -0.41968512535095215, 1.4343830347061157, -0.3314504027366638, -1.1858896017074585, -0.9839255213737488, -0.9842787384986877, -0.5318474769592285, 0.3077446520328522, -1.0090727806091309, 0.38462522625923157, -0.26729077100753784, 1.7715214490890503, 0.353781133890152, 0.4019354581832886, -0.3240729570388794, 0.10455423593521118, -0.9444419741630554, -0.3147071897983551, 0.036206744611263275, 1.59407639503479, -0.7364616990089417, -1.0409107208251953, -2.3055644035339355, 0.7509166598320007, 1.4732892513275146, -1.435297966003418, -0.3104858100414276, 1.260208249092102, 1.2995014190673828, 0.31250277161598206, 1.7469635009765625, -1.1030142307281494, 0.4618168771266937, -0.14777058362960815, -0.6142064332962036, -1.509113073348999, -2.173119068145752, 0.5351980328559875, 0.3770986497402191, 0.2304346114397049, 0.6781527996063232, 0.2873905301094055, 1.1295469999313354, -0.09531646221876144, -1.341036081314087, -0.5353781580924988, -2.085286855697632, 0.73525470495224, -0.38934391736984253, 0.4845520853996277, 0.0547439381480217, -0.6668934226036072, 0.4236820340156555, -0.8482156991958618, 0.6056892275810242, 0.9615129232406616, 1.3312904834747314, -0.43922653794288635, -0.2621587812900543, 0.4074104130268097, -0.30297988653182983, 0.14990855753421783, 0.21177223324775696, 0.9499468803405762, -0.4053858518600464, 0.2464209794998169, 0.903381884098053, 0.6118171811103821, -0.12131476402282715, -0.8720882534980774, -0.5609016418457031, -1.3500633239746094, 0.8996198177337646, 1.0939446687698364, -1.2576388120651245, 0.7197461128234863, 0.10428935289382935, -0.7113116979598999, -0.19896872341632843, 1.7083779573440552, 0.1926867961883545, 0.6367599368095398, -0.6653583645820618, 1.0614182949066162, -0.21032442152500153, -0.8574895858764648, 1.398177981376648, 0.049762021750211716, -0.4869414269924164, -0.0011428145226091146, 1.075824499130249, 1.1185446977615356, -0.7105198502540588, 0.19033099710941315, -1.2705366611480713, 0.7729089260101318, 0.23751391470432281, 0.7877539992332458, 0.9467383623123169, -0.3567326068878174, -0.2965439260005951, -0.6169489622116089, 0.5570999383926392, -1.5506342649459839, 0.31708404421806335, -0.9347200989723206, 0.6228250861167908, 1.4813770055770874, 0.22666537761688232, 1.723010778427124, 1.274000644683838, 1.1664141416549683, -0.15625648200511932, 0.33840513229370117, -2.0626211166381836, -1.668349266052246, 0.35007211565971375, 2.90602707862854, -0.2701975405216217, 1.6814452409744263, 2.0219621658325195, 1.0796403884887695, -0.741690456867218, -0.09611176699399948, 0.9178999066352844, 1.6308499574661255, -1.2657550573349, -1.414590835571289, 0.642008364200592, -0.18956521153450012, -0.8067840337753296, -0.22912995517253876, 1.6370899677276611, -2.103903293609619, 1.530300259590149, 0.1411503702402115, -0.17277590930461884, -1.5262216329574585, -0.23199450969696045, -0.6282366514205933, -0.19291722774505615, -1.1573868989944458, -0.022372758015990257, 0.7138121128082275, -0.855497419834137, -1.1249831914901733, 0.11000477522611618, -0.4612286388874054, 0.2386428415775299, 0.9880061745643616, -2.3107101917266846, -0.6415932178497314, -0.9987756609916687, -1.173376202583313, -0.6481629610061646, 0.5678768157958984, 0.6954705715179443, 0.08465275913476944, 0.6081023812294006, -0.3265381455421448, -0.9505439400672913, 0.2971201539039612, -0.26739606261253357, 0.0067789750173687935, 0.07692917436361313, 1.3727220296859741, -1.1433063745498657, -0.19126644730567932, 0.9699869751930237, 1.0186530351638794, 0.4492076337337494, 1.0903289318084717, -0.7951903343200684, -0.05123390257358551, 0.21719862520694733, -0.8644440770149231, -0.5866620540618896, -0.8267918825149536, -1.6198714971542358, -0.6330087780952454, 2.0181801319122314, 0.48464855551719666, 1.6487690210342407, 1.0937944650650024, 1.7103381156921387, -1.6118340492248535, -0.08598271012306213, 1.4127442836761475, 0.5682234764099121, -0.2376025915145874, -0.17623591423034668, 1.0731780529022217, -1.9580588340759277, 0.17107461392879486, -0.04607176408171654, 0.14845332503318787, -0.092258520424366, 1.4507579803466797, 1.295054316520691, 0.6988315582275391, 1.5410646200180054, -0.14931237697601318, -0.5145723223686218, -1.7981171607971191, 0.07832837849855423, -0.029102444648742676, 0.44423994421958923, -0.9021627902984619, 0.354275643825531, 0.28213903307914734, -0.08006083220243454, 0.37419235706329346, -0.8074791431427002, -0.6743730306625366, 0.417712539434433, -0.06861499696969986, 0.061910372227430344, -0.15911917388439178, -0.28195488452911377, 0.5803960561752319, -1.4951101541519165, 0.3302401900291443, 1.2405647039413452, 1.5599305629730225, -0.2171984761953354, -1.5966625213623047, 0.12126369774341583, 0.9707740545272827, 0.6815146207809448, -0.6787930130958557, 0.27069973945617676, -0.8942116498947144, 2.053887367248535, -0.11118384450674057, 1.1880186796188354, 1.3389649391174316, 1.1198900938034058, -0.17872506380081177, -0.2866133749485016, 1.6567333936691284, 0.40413516759872437, -1.2273133993148804, -0.12723511457443237, -2.038325071334839, 0.7287755608558655, 1.0980912446975708, -0.2623191475868225, 0.23467057943344116, -0.5097792744636536, 1.5654538869857788, -0.4130944311618805, -1.265260100364685, -0.869214653968811, 0.209416925907135, -1.1992148160934448, -0.690359354019165, -0.05335832014679909, 0.19414874911308289, -0.5785900354385376, 0.4548449218273163, 0.8729485273361206, -1.281283974647522, 0.7206256985664368, -0.59722900390625, -0.9186279773712158, 1.5453948974609375, 1.932088851928711, 0.9785294532775879, -0.42982229590415955, 1.0003032684326172, -1.4755067825317383, -0.5137038230895996, 0.051351550966501236, 0.1389590948820114, 0.9634844660758972, 1.0940015316009521, 0.18836189806461334, 0.38988420367240906, 2.288898229598999, -0.5339096784591675, 0.12714023888111115, 0.7033038139343262, 2.2373855113983154, -0.5333929657936096, 0.9967120289802551, -0.8557827472686768, -0.8201941251754761, -1.606884479522705, 0.011886159889400005, 0.41459718346595764, 1.648489236831665, 3.0205719470977783, -1.12461256980896, 0.30238837003707886, 1.0328459739685059, 0.8298986554145813, -0.29900428652763367, -1.6778056621551514, 1.4043508768081665, -0.16609473526477814, 1.764055848121643, 0.7779326438903809, -2.345919609069824, 1.0188935995101929, 0.10463178902864456, 0.561573326587677, 1.2401416301727295, 0.8131768703460693, -0.4325678050518036, -1.4484615325927734, 1.506205677986145, -0.6349970102310181, -1.6427258253097534, -1.0746854543685913, -1.2163891792297363, -0.14159716665744781, -1.0230633020401, -1.1412733793258667, 0.2467975914478302, -0.2250991016626358, -1.9518961906433105, -1.042421817779541, 0.516028881072998, -0.013856315985321999, 0.1527620553970337, -0.24450761079788208, 1.2690019607543945, 0.5713096857070923, 1.0593256950378418, 0.2664848864078522, 0.9982742667198181, 0.11702210456132889, -0.7036781311035156, -0.15256306529045105, -0.30867594480514526, -0.2226206660270691, -2.25998592376709, 0.6180863976478577, 1.9561394453048706, -0.027627263218164444, 1.0280308723449707, -0.4865208566188812, 0.9218206405639648, -1.037606954574585, -1.893619418144226, 1.4344061613082886, -0.19863003492355347, -0.5286346077919006, 1.1621079444885254, 2.0965254306793213, 2.1557199954986572, -0.3599017858505249, 1.0428869724273682, -0.7005903720855713], [0.3255039155483246, -0.5994467735290527, 0.006965010426938534, 0.8521348237991333, 0.35692837834358215, 0.19448086619377136, 0.32182514667510986, -1.8809168338775635, -0.8931611180305481, -0.30195462703704834, -0.12932342290878296, -0.2896263897418976, -0.07382374256849289, 0.4342363476753235, -0.7011792063713074, 0.002265112940222025, 0.4203294515609741, 0.11172103136777878, -0.19651535153388977, 0.021168740466237068, 1.5503721237182617, -0.2714618146419525, -0.47813940048217773, -1.4054174423217773, 0.44455432891845703, -0.8367936015129089, 0.16350896656513214, 0.8719131946563721, 0.5949247479438782, 0.2808871269226074, 0.45910218358039856, -1.0411955118179321, 0.6245899200439453, 0.045864056795835495, -0.019064852967858315, -0.454039990901947, 0.468230277299881, -0.5327886939048767, 0.02111436054110527, -0.5799304246902466, 0.0014221376040950418, 1.331593632698059, -1.2048407793045044, -0.5488829612731934, 0.955578625202179, 1.0223970413208008, -1.4381139278411865, 1.7447843551635742, -0.40734153985977173, -1.5015028715133667, 1.860731840133667, -0.05060303211212158, 0.9446664452552795, 0.18581095337867737, 1.5075782537460327, -1.4056766033172607, -0.6815534830093384, 0.3923526406288147, 1.3237922191619873, 1.286298394203186, 1.412207841873169, -0.2673979103565216, -0.5382537841796875, -0.8301118016242981, 1.2142454385757446, -0.24393334984779358, 1.127216100692749, -0.3131585717201233, 0.2480451613664627, 0.7468547821044922, 0.199533611536026, 0.9142543077468872, -0.9426458477973938, 0.21477743983268738, -0.6723957061767578, -1.2323639392852783, -0.256916344165802, 0.701034665107727, 1.5868693590164185, -1.6747713088989258, 0.7753979563713074, 0.9702757000923157, 0.5621196031570435, 0.5177550911903381, 0.6134135127067566, -0.9622024297714233, 1.1514489650726318, 0.024630742147564888, 1.6469193696975708, 0.7904251217842102, 0.4173020124435425, -0.6590133905410767, 0.9430992603302002, -0.8168590664863586, 0.5395240187644958, -1.0676685571670532, 0.8634406328201294, 0.34588125348091125, 1.253416895866394, 0.2611178457736969, 0.4094770848751068, 1.0530961751937866, -1.2660164833068848, 0.2323804348707199, -1.9219317436218262, -1.3167983293533325, 0.702303946018219, 0.6749222278594971, -0.3458591401576996, 0.5300849676132202, -0.9961490035057068, 1.5038461685180664, 0.34338781237602234, -0.8989781141281128, 0.5216516852378845, 0.4050178527832031, -1.3407385349273682, 0.19538164138793945, -0.12939293682575226, -0.9809356331825256, 0.016214575618505478, 1.237833023071289, 1.7507597208023071, 0.6024044752120972, -1.1727005243301392, 0.6055961847305298, -0.1355830430984497, -0.23731452226638794, 0.46925610303878784, 0.4340217709541321, -0.26429638266563416, 0.6979944705963135, -1.394525170326233, -2.247199773788452, 0.7636895775794983, -1.5340449810028076, -1.4459869861602783, -0.8403773903846741, -0.8004506230354309, 0.43246760964393616, -2.0070574283599854, -2.0379295349121094, 0.713989794254303, -0.18348541855812073, -0.24640101194381714, -0.6665047407150269, 1.1827186346054077, 0.3291952311992645, 0.38820165395736694, -1.109494924545288, 0.5067141652107239, -0.4351448714733124, 0.4483663737773895, -0.17568349838256836, 0.5876102447509766, 1.5111979246139526, 1.4046927690505981, -2.3927507400512695, 0.1740819215774536, -1.6345250606536865, -0.34531787037849426, 1.1989768743515015, -0.04691363498568535, 0.6879918575286865, -1.7472782135009766, -0.24398158490657806, -0.721285879611969, -0.37302491068840027, -0.23101696372032166, 0.39989641308784485, -0.8333802223205566, 0.08234754204750061, 0.4196837544441223, 1.2984164953231812, -0.2809249460697174, -0.2368147075176239, -0.2543574869632721, -1.0400187969207764, 1.213403344154358, -0.2556789517402649, -1.4060875177383423, -0.25075748562812805, 1.798522710800171, -0.18513242900371552, 0.5619803071022034, 1.1386767625808716, -0.7778347134590149, -1.8063616752624512, -0.6232830882072449, -0.13676130771636963, -0.9649509787559509, -0.6730951070785522, -0.8003970384597778, 0.7582593560218811, 0.06009290739893913, -0.8575767278671265, -0.5503720641136169, 0.020681368187069893, 1.3627839088439941, 0.09520550817251205, 1.2208192348480225, -0.23655390739440918, 1.3952915668487549, -1.3785651922225952, -1.0038297176361084, -1.1770172119140625, -0.1202302798628807, 0.9925146698951721, -0.8525253534317017, 0.5670482516288757, 0.06070986017584801, -0.7156014442443848, 1.637866497039795, -0.20923173427581787, 0.32285767793655396, 0.08854680508375168, 0.4275941550731659, -0.9604395627975464, -0.16745266318321228, -0.08953456580638885, -2.43469500541687, 0.09040682017803192, -0.13143539428710938, 0.42593809962272644, 0.21263277530670166, -0.4975918233394623, -2.2461071014404297, 1.9434669017791748, 0.2893202602863312, -0.40510818362236023, 1.2459381818771362, -0.9961423277854919, -0.0962020754814148, -0.45783019065856934, -0.0453314483165741, 0.12235023826360703, -1.1198300123214722, 0.7898939847946167, 0.9142605662345886, 0.28503143787384033, 0.774527370929718, 0.2092995047569275, -0.5956801772117615, 0.7184820771217346, 1.8520514965057373, -0.8355734944343567, 1.9645131826400757, -0.674651026725769, -1.200453281402588, 0.407825231552124, 0.39748242497444153, 0.25935935974121094, -0.5822347402572632, -0.6280900239944458, 0.6339877843856812, -0.29183128476142883, 0.6086734533309937, 0.12208317220211029, 1.3689740896224976, 0.7793843150138855, 0.0735604465007782, 0.9381374716758728, -1.9357303380966187, 0.012385005131363869, -0.06172686815261841, 0.7327426075935364, 0.34608030319213867, 0.2365926206111908, 0.47401371598243713, 2.6957759857177734, 1.215947151184082, -0.42316555976867676, 1.1554291248321533, 0.7801417708396912, 1.138431191444397, 1.086499810218811, -0.9487668871879578, 0.04129623621702194, 2.15932559967041, -0.29303717613220215, 1.5075410604476929, -0.8985401391983032, -0.989222526550293, 1.112980604171753, -0.6455307602882385, 1.250514268875122, 0.3256641924381256, 1.1033631563186646, 0.2760591208934784, 0.3603459894657135, -0.9536929130554199, 0.05360357090830803, -0.553661584854126, -0.1519067883491516, -0.09491919726133347, -1.7166213989257812, 1.8060849905014038, 1.532101035118103, -0.4558875262737274, -1.1286778450012207, 1.0310170650482178, 1.0783214569091797, -1.1765590906143188, -1.2199256420135498, -0.42777296900749207, -1.4555583000183105, -0.5595347881317139, -1.8254343271255493, -0.8580303192138672, 1.547899603843689, 1.7335907220840454, 0.8066811561584473, -2.03562331199646, 1.6078736782073975, 2.3737430572509766, 0.025578230619430542, 0.8507869243621826, -0.2787034213542938, 0.24490021169185638, 0.9348880052566528, -0.5766889452934265, -0.8899961113929749, -1.0851612091064453, 1.264434814453125, 0.4988531470298767, -0.11281318217515945, 2.428260326385498, 0.42756760120391846, 0.7175596952438354, 0.891998291015625, 0.41600850224494934, 1.4205822944641113, 0.6441649198532104, 0.45197001099586487, -0.36210906505584717, 0.8161883354187012, -0.51316899061203, -1.0791569948196411, 0.5209962129592896, 1.620218276977539, -1.1428446769714355, 0.41920411586761475, -0.9446857571601868, 0.8767200708389282, 0.5335978269577026, -1.235929012298584, 0.3254150450229645, -0.7753903269767761, -0.2963162064552307, -0.5780949592590332, -0.787322998046875, -0.5324952006340027, 0.7076523900032043, 0.1251162588596344, -0.6032422780990601, -2.2844505310058594, 1.7555153369903564, -0.46264299750328064, 1.215021014213562, -0.6060404777526855, -0.1586986482143402, -2.2948570251464844, 0.007120884023606777, -0.924116849899292, -0.6760310530662537, -2.540283441543579, -0.3573022484779358, -1.1084908246994019, -0.29746395349502563, -0.6722815632820129, -0.2765927314758301, -0.6576309204101562, 0.16464084386825562, 1.291916012763977, -0.2167491465806961, -0.62497878074646, -1.0011942386627197, -3.0371129512786865, -0.35871028900146484, 1.023369312286377, -0.0941886231303215, 0.4764839708805084, 0.8709884881973267, -0.35708722472190857, 1.0038717985153198, -0.9597503542900085, 0.08774534612894058, 1.9215458631515503, -0.17234832048416138, -0.5338490009307861, 0.8962303400039673, 0.5504348874092102, 1.3126837015151978, 1.1538907289505005, 0.5806339979171753, -0.26991742849349976, -1.9237293004989624, 0.10699313133955002, -0.5385591387748718, -1.1716816425323486, -1.68626070022583, 0.5044226050376892, -1.7674342393875122, 1.318013072013855, -0.4767301678657532, -0.4868840277194977, -0.3247072994709015, 0.2448129951953888, -0.8993940949440002, 0.4013303816318512, 1.9985460042953491, -0.6172761917114258, -0.8256849050521851, -0.5703298449516296, 0.4595493674278259, -0.987933874130249, 0.12334294617176056, 0.472919762134552, -0.42987653613090515, 1.6059154272079468, 0.04093040153384209, -0.6327372789382935, -1.2330536842346191, -1.4974796772003174, 0.048960715532302856, -0.5804528594017029, -0.9043009281158447, -0.09705352783203125, -1.4651846885681152, 1.006395936012268, 1.587944507598877, 0.2536714971065521, -0.5569140315055847, 1.240606665611267, -0.29774796962738037, 1.0728904008865356, -1.8188248872756958, 2.0590431690216064, -1.1503695249557495, 1.085412859916687, -1.1841340065002441, 1.10397207736969, 0.18395937979221344, -1.2213139533996582, -0.5482425093650818, -2.159451484680176, 0.5306053757667542, -1.55726957321167, -0.24354109168052673, 1.3425374031066895, 0.5231969356536865, -0.7210016250610352, -1.5470221042633057, 1.1293708086013794, 1.1042202711105347, 1.997748851776123, 0.06954716145992279, -0.4431610703468323, -0.3586764931678772, -0.2801550328731537, 0.4480060636997223, -0.7502090930938721, 0.0493006631731987, 0.6921641230583191, 1.6162687540054321, 0.01998661831021309, 0.9860615134239197, 1.1714013814926147, -1.3913217782974243, 0.8970217108726501, 0.000991690787486732, 1.5312899351119995, -0.9426435828208923, -0.08106018602848053, 1.0970791578292847, 0.23921337723731995, -0.852448046207428, 0.13783398270606995, 0.18144994974136353, 0.46983128786087036, 1.028600811958313, -0.8374360203742981, 0.22751502692699432, -0.9519489407539368, 1.4192054271697998, -0.06304755061864853, -0.4115487337112427, 2.117039918899536, 0.1227642148733139, 0.11401504278182983, -0.5420262813568115, 0.23740459978580475, -1.4003230333328247, -0.4117198884487152, 1.0548021793365479, 0.49834558367729187, 0.14656712114810944, -0.07945990562438965, -1.415467619895935, -0.6720084547996521, -1.3113691806793213, -1.6538324356079102, 0.2355513721704483, -0.6373564600944519, 0.6449488997459412, 1.0523864030838013, -0.01089964434504509, -1.0098211765289307, -0.41632768511772156, -0.06994070112705231, -0.6520269513130188, -0.7854205369949341, 0.3464697599411011, 0.2776561379432678, 0.4172060787677765, -0.5752621293067932, 0.9768277406692505, 0.4151361286640167, -0.4826370179653168, 0.26161596179008484, 1.6522711515426636, 1.5963906049728394, -0.4999280869960785, -1.211593508720398, 0.9622228741645813, 1.3746174573898315, -1.0285357236862183, -0.1877117156982422, 1.0087414979934692, 0.14410682022571564, -0.32055601477622986, -0.5355504155158997, -0.5298494696617126, 1.0759698152542114, -0.18310464918613434, -0.2045646756887436, 0.8445929288864136, -0.8590761423110962, 0.8717853426933289, 0.941979169845581, 2.016810655593872, -1.1603541374206543, -0.2613763213157654, 0.008741997182369232, -0.6967790722846985, 0.5193039774894714, -0.4312630891799927, -0.30412474274635315, 0.6307268738746643, -0.06129482761025429, 0.38446763157844543, 0.8985873460769653, 0.551950216293335, -1.235152244567871, -0.2940233647823334, -1.5004514455795288, 0.5494644045829773, -0.43967190384864807, 0.804646909236908, 0.09413401037454605, -1.016594648361206, 0.07010489702224731, -1.2625453472137451, -0.05825384706258774, 1.3385854959487915, -1.1699200868606567, 0.9091004729270935, -0.20879320800304413, -0.9826674461364746, -0.17910896241664886, -0.5099478960037231, 0.5739389061927795, 0.17652037739753723, -0.2394181787967682, 0.31230708956718445, 0.48880210518836975, 2.1036787033081055, -0.9549522399902344, -1.2836368083953857, -1.6118797063827515, 0.15273141860961914, -1.3683770895004272, -1.428000807762146, 0.08616776019334793, -0.6272311806678772, 1.567730188369751, 2.056671619415283, -0.7892197966575623, 0.567034125328064, -0.30557382106781006, -0.1307588815689087, -1.1573723554611206, -0.21322818100452423, -0.914122998714447, -0.7153101563453674, 0.7644869089126587, 0.8991394639015198, 0.1340988427400589, 0.208546444773674, -0.21274247765541077, -0.007322486024349928, -0.5974871516227722, -0.23153138160705566, -0.4675973057746887, -0.6291818022727966, 0.49474090337753296, 0.5776750445365906, 0.6847975850105286, 1.0209686756134033, -0.4069090187549591, -0.02699194848537445, -0.21406880021095276, -0.004689941648393869, -1.0190622806549072, 0.16040579974651337, 0.41798847913742065, 0.4777570366859436, 0.3450043201446533, -0.311960905790329, 1.9721331596374512, 0.5047779679298401, -2.8794827461242676, 2.0790493488311768, 1.0065711736679077, -0.5058348178863525, 0.008246936835348606, 0.3890211880207062, 1.147301435470581, 1.3375300168991089, 1.338578701019287, 1.0623451471328735, 0.2545470595359802, -0.3452172875404358, -0.00873598176985979, -0.6444674134254456, -1.48116135597229, -0.4353150427341461, -0.39383286237716675, -0.7286123037338257, 0.707351565361023, -0.7452822923660278, 0.4319106340408325, -1.2178024053573608, -0.018639016896486282, 0.11960147321224213, 1.4341005086898804, -0.1968427449464798, 0.6173470616340637, -1.7890397310256958, -0.4318149983882904, 0.17055121064186096, -0.30750325322151184, 2.2529401779174805, 0.6687251329421997, 0.9283196330070496, -1.095192551612854, 1.1349457502365112, -0.9225611090660095, -0.10926484316587448, -0.5052586197853088, -0.1440635621547699, 1.3672503232955933, -1.1931108236312866, 0.8647070527076721, 0.9887745976448059, -0.406745582818985, 1.7516894340515137, 0.40393131971359253, 0.22112581133842468, 0.618920087814331, -0.6731086373329163, 0.129821315407753, 0.8738502264022827, 0.0742228776216507, 0.9793795943260193, 0.8436930179595947, -1.3932621479034424, 0.9674186110496521, -0.15434221923351288, -0.7337404489517212, 0.03493630886077881, 0.25009843707084656, 0.9569673538208008, -0.38388627767562866, 0.47604334354400635, 0.5613162517547607, -2.0300145149230957, 0.03215648978948593, -0.48688772320747375, -0.7456936836242676, -0.1558334082365036, 2.668278694152832, -1.106929063796997, 0.10934393852949142, -1.3956249952316284, -0.49506258964538574, -0.5989753603935242, 1.9475089311599731, 1.7763575315475464, 0.8075183033943176, -1.4290108680725098, 0.42776504158973694, 1.0762302875518799, -1.3302819728851318, -0.8265944719314575, 1.0992002487182617, 0.8081135153770447, -0.4621787965297699, -1.352761149406433, 0.4055747389793396, 0.4295704960823059, -0.47803232073783875, 0.41327959299087524, 0.8749166131019592, 0.0476800762116909, 1.3795151710510254, 0.26319366693496704, -0.6433688402175903, -0.44562748074531555, 0.2621139883995056, 0.09231072664260864, 1.1632596254348755, 0.8738795518875122, 0.8744992613792419, 0.1947356015443802, 0.7954458594322205, -1.141810655593872, 0.20270323753356934, -1.0220133066177368, -0.11674335598945618, -1.07585871219635, 0.9649323225021362, 1.3732634782791138, 0.617259681224823, 0.29380181431770325, -1.513443112373352, -0.267678827047348, -0.7251440286636353, -0.019829584285616875, 1.2047605514526367, 2.5593714714050293, 0.19443583488464355, 1.0658763647079468, -0.23451682925224304, 1.4169063568115234, -1.729761004447937, -1.7460763454437256, 0.6946858763694763, -0.3821185231208801, 0.9552671909332275, -0.6249128580093384, -1.0936394929885864, 0.0226524006575346, -0.026288453489542007, 1.3155311346054077, 0.6006507873535156, -0.07920925319194794, 0.44454121589660645, -0.11510945111513138, 0.2783887982368469, 0.28286629915237427, 0.32497307658195496, 1.7640281915664673, -0.8581418991088867, 0.16784122586250305, -0.6733144521713257, -1.4455184936523438, 1.5593096017837524, 1.7960246801376343, 1.047119379043579, -0.7146068811416626, -0.8402567505836487, 1.388569712638855, 0.04491908475756645, 0.6139851808547974, -1.0612133741378784, 0.5599580407142639, 0.12306635081768036, 0.2814828157424927, 1.5036356449127197, 0.004771944135427475, -1.551683783531189, 0.6818022727966309, 0.805147111415863, -0.12387354671955109, -0.7048271894454956, 0.36043646931648254, -0.1685517281293869, -0.9989731311798096, 0.9924296140670776, 1.8426885604858398, -0.12023542821407318, 0.3593083322048187, -1.7655742168426514, 1.9054864645004272, 1.1175144910812378, -0.9626444578170776, -0.4384782910346985, -0.4249992072582245, 1.592948317527771, 0.4070169925689697, -0.4868837296962738, -1.4549144506454468, 0.3119368851184845, 0.702060341835022, 1.151724934577942, -2.118281841278076, -0.09591883420944214, -1.2102137804031372, -0.9131527543067932, 0.3033228814601898, 1.3324073553085327, -1.1321848630905151, -0.7255260348320007, 0.2558126151561737, -2.254875659942627, -1.457890272140503, -0.4388340413570404, -0.1770036220550537, -0.1868799775838852, 0.6952138543128967, 0.8309481739997864, 0.6142563223838806, 0.4802868962287903, 2.533869981765747, 1.6305537223815918, 0.12427753955125809, -1.9470067024230957, -0.07049208134412766, 0.7561137080192566, 0.8091390132904053, 0.9450051784515381, -1.3449273109436035, 1.1566017866134644, -0.5705490708351135, -1.3533121347427368, 1.4228914976119995, 0.292838990688324, -1.4693825244903564, 1.151303768157959, 0.3906533420085907, -0.44971463084220886, -0.19543133676052094, 0.13974528014659882, -1.3412039279937744, -1.2395679950714111, -1.0910611152648926, -0.6204162836074829, -0.9896867871284485, 0.5944596529006958, -0.21861597895622253, -0.7015584707260132, -0.03090275451540947, 0.026891998946666718, -0.4991261661052704, 1.4629793167114258, -0.3134145140647888, 0.12262311577796936, 0.5595415830612183, 0.8151474595069885, 0.8418315649032593, 0.16266591846942902, -0.10875856131315231, -0.732228696346283, 0.6587372422218323, -0.433206707239151, -0.29489123821258545, 1.4425090551376343, 0.11300607025623322, 0.897530734539032, -0.34941375255584717, -0.9437239170074463, -0.7191027998924255, 1.1738827228546143, 0.9610604643821716, -0.044240403920412064, -1.8611061573028564, 0.8617688417434692, -1.311370611190796, 0.16406656801700592, 2.237880229949951, 1.6313982009887695, 0.5085226893424988, 2.211000442504883, 2.382288932800293, 0.19317150115966797, -1.3581933975219727, 0.20236006379127502, 0.6434620022773743, 0.6776973009109497, 0.18454097211360931, 2.0103187561035156, 1.407111406326294, 0.9840559363365173, 0.6458421349525452, 0.6794770956039429, 0.22276553511619568, -0.7030047178268433, -0.27300190925598145, 0.8096834421157837, 0.89711594581604, -0.1431661695241928, 0.4497097134590149, -0.16358694434165955, -0.8697203993797302, -1.1271241903305054, -0.5779587030410767, -1.177816390991211, -0.13100436329841614, 0.5366009473800659, -0.7694900631904602, 1.4246264696121216, -0.5866795778274536, 0.30158481001853943, -1.0594896078109741, -0.5282424092292786, -0.3605984151363373, 0.7683277130126953, -2.1436071395874023, 2.2113230228424072, 0.17886486649513245, -1.6866453886032104, -0.3898172378540039, -1.5979214906692505, 0.3440805673599243, -1.473501443862915, -0.3500747084617615, 0.3300929069519043, 2.04805326461792, -0.8833613395690918, -0.09205184131860733, -0.537223219871521, -1.093750238418579, 0.24707123637199402, 0.7103415727615356, 1.5218898057937622, 0.2100515067577362, -1.2294437885284424, -0.7333755493164062, -0.46346014738082886, -0.3793628513813019, 0.06219267100095749, 0.11641961336135864, 1.4611812829971313, 0.3310241103172302, -0.16889572143554688, 1.566458821296692, 0.8681976199150085, -1.4836971759796143, -0.0615994967520237, 0.12028102576732635, 0.05231752619147301, 0.09988490492105484, -0.7663353085517883, -0.44344907999038696, -0.11427053064107895, -0.32010212540626526, 0.6794296503067017, 0.4061330556869507, 0.028276240453124046, -1.6563810110092163, 0.8213203549385071, -0.7119083404541016, -0.14601151645183563, 2.049896478652954, 0.11066377907991409, -1.4498112201690674, 1.6111434698104858, -0.08219635486602783, -0.21372129023075104, -0.4442974328994751, 0.5439183115959167, -0.4771200716495514, 0.0038505159318447113, 0.22829413414001465, -0.19169841706752777, 1.0989681482315063, -1.0918382406234741, 0.6081574559211731, 0.8564938306808472, 0.12008167803287506, 0.23818260431289673, -0.6236227750778198, -1.5808959007263184, 0.06746485084295273, 0.9789500832557678, -1.8691779375076294, -1.405170202255249, 0.032772377133369446, 0.2714496850967407, 1.2198410034179688, 0.2852250337600708, 0.9164751172065735, -2.282456874847412, 1.0675294399261475, -1.4026542901992798, 1.6755346059799194, 0.4400985538959503, -1.1934537887573242, 0.7246072292327881, 0.8365257382392883, 0.24194295704364777, 1.0125237703323364, -0.3292168080806732, 0.24301327764987946, 1.4448283910751343, 0.2581027150154114, 0.09352525323629379, -0.346782386302948, 0.3547137379646301, 1.1776525974273682, 0.000765934877563268, 0.976936936378479, -0.9025018215179443, -0.7226345539093018, 0.09573327749967575, -1.6487021446228027, -1.5737664699554443, 2.148549795150757, -0.8029319047927856, 0.19220282137393951, 0.8165112137794495, 1.0007755756378174, -0.38371390104293823], [-0.792078971862793, -1.6541361808776855, -1.4799057245254517, -0.9050441980361938, 0.7594202756881714, -0.1445467323064804, -0.8374817967414856, 0.09312287718057632, 0.6021410822868347, 1.6590255498886108, -0.9057253003120422, -0.7626832723617554, 1.0733393430709839, -1.645348072052002, -1.1305242776870728, -0.7105608582496643, 0.6712743043899536, 1.3480054140090942, 0.6324798464775085, -0.448040634393692, -0.5974830985069275, -0.4612947106361389, 1.1802159547805786, -0.1159483790397644, -1.3178761005401611, -1.5985102653503418, -0.5386162996292114, -2.067098617553711, 1.1860626935958862, 0.15471318364143372, 1.4447168111801147, 1.0002917051315308, -1.7680107355117798, -0.6015648245811462, 0.20582430064678192, -0.47492343187332153, 0.08029923588037491, 0.8314439058303833, 0.2738475799560547, 0.7441061735153198, 0.8383520841598511, -0.5281211733818054, -1.111510992050171, -2.0240087509155273, -1.1188504695892334, 0.22976729273796082, -0.3805280923843384, 0.5178876519203186, -2.196622371673584, 1.8822269439697266, 0.8594763875007629, -1.7114757299423218, -0.8770765662193298, 0.3774053156375885, 3.312206268310547, 0.20957015454769135, 0.4000670611858368, 1.095101237297058, -0.2594631612300873, 0.5762186646461487, -0.4860914647579193, 0.5108393430709839, 0.7037980556488037, -0.6060245037078857, 0.06711325794458389, -0.8995546698570251, -0.3290168344974518, -0.0442238412797451, -0.7456535696983337, 0.033003777265548706, 0.28278133273124695, 0.5961860418319702, 1.0538760423660278, -0.1874370127916336, 0.9529605507850647, -0.793397843837738, -0.854634165763855, -0.27493155002593994, 0.6652109026908875, 0.6339215040206909, -1.1198829412460327, -0.16185803711414337, -0.0806550532579422, 0.06614413112401962, 0.5632663369178772, -2.174323081970215, 1.379189133644104, 0.12216541916131973, 0.8329780697822571, -0.025863075628876686, -0.39730769395828247, 0.8772749900817871, -0.05170265585184097, -0.9843232035636902, -0.004735688678920269, -0.058682724833488464, -0.6952975392341614, 1.6546025276184082, 1.8492822647094727, 0.46780991554260254, -0.3535762131214142, 0.20403756201267242, 1.5126266479492188, -0.14499004185199738, -0.2937067449092865, 0.755009651184082, -1.290693998336792, 0.1291801482439041, -0.00355238257907331, 0.21099728345870972, -1.8319011926651, -2.0242929458618164, 0.5338260531425476, -1.1150996685028076, 0.5581960678100586, -0.2670510411262512, -0.7129461765289307, 0.728408932685852, -1.3793684244155884, 0.821568489074707, -0.14486651122570038, 0.23010721802711487, 0.668599545955658, -1.6986169815063477, -0.4136323630809784, -0.9149088859558105, 1.2422900199890137, 0.0024591651745140553, -1.3299121856689453, 1.5240464210510254, -1.1228913068771362, -0.9591836929321289, -0.8578383326530457, -0.5763404369354248, -1.2134020328521729, -0.20245489478111267, -0.3842450678348541, -0.1346886157989502, 1.2798895835876465, -0.08966878801584244, -0.7343384027481079, 0.34814131259918213, 2.1346635818481445, 0.7986491918563843, -0.4714362621307373, 0.7832275629043579, -1.1666276454925537, 0.29414618015289307, 1.5929253101348877, -3.6403162479400635, 0.3542693257331848, -0.5122525691986084, 0.6086426973342896, 0.6834412813186646, 0.4664890170097351, 2.454500675201416, -1.5222185850143433, -0.5103798508644104, -0.5254859924316406, 0.12718896567821503, -0.287197083234787, -1.6404999494552612, -0.0019013300770893693, -0.44654279947280884, -1.404643177986145, -0.5843607783317566, 1.066580057144165, -0.8858059644699097, -0.5484951138496399, 1.2690739631652832, 0.7020989060401917, 0.1374131292104721, -1.1301817893981934, -0.40233588218688965, -2.1079299449920654, 1.1117984056472778, -1.3058717250823975, 1.1805487871170044, 1.7502167224884033, -1.159314751625061, 0.7215808033943176, 0.7843847274780273, -0.5966601967811584, 0.37849947810173035, 0.905707597732544, 0.7497352361679077, -0.20286491513252258, -0.5633836984634399, -0.7493188381195068, -1.348995327949524, 0.27686038613319397, 0.4530192017555237, 1.0280460119247437, -1.2540721893310547, -0.08061189949512482, -1.4491336345672607, -1.242529273033142, 1.8339462280273438, 1.387279748916626, 0.7891870737075806, -0.4921571910381317, -0.09547149389982224, 2.044080972671509, 0.5066141486167908, -0.11065850406885147, -0.9037328362464905, -0.21463213860988617, 0.8411394953727722, -0.4929303824901581, -0.2575591504573822, -0.21560004353523254, 1.1794523000717163, 0.541547417640686, 0.40963977575302124, 0.22291746735572815, -1.4840909242630005, 0.364689439535141, -0.6339324116706848, 1.9649183750152588, -0.8880284428596497, -0.338985413312912, -0.3633524477481842, 0.5465582013130188, 0.04849935695528984, 0.9720168709754944, -0.19883863627910614, 0.5074671506881714, 0.8566662073135376, 0.6947897672653198, 1.4901288747787476, -0.055969417095184326, 0.29143983125686646, 0.9224807620048523, 0.45106956362724304, 0.43941545486450195, 0.034619126468896866, -0.5881207585334778, 0.10373692959547043, -0.6767122149467468, 0.1390639990568161, 0.8496127128601074, 0.04313446581363678, 1.620909571647644, 0.06189243495464325, -0.9464206099510193, -0.6206325888633728, 0.20750826597213745, -1.1377061605453491, 0.26232048869132996, -0.18990103900432587, 1.5413196086883545, 0.4243491590023041, 2.1757357120513916, 0.6335828900337219, -0.1834469884634018, 1.0400958061218262, 1.6677825450897217, -0.8346813321113586, -0.530275285243988, 0.5536445379257202, -0.6739687919616699, -0.5077884197235107, 0.07252975553274155, 0.9940469861030579, -0.4412541389465332, -0.9348331689834595, 1.2161833047866821, -1.047898530960083, -0.7314647436141968, -1.4481912851333618, 1.2386263608932495, -0.2590087950229645, 0.7417487502098083, -1.0437901020050049, -0.22319923341274261, -0.3991945683956146, 2.159768581390381, 0.3294016122817993, 0.19440482556819916, 0.22050794959068298, 1.0882340669631958, -1.751870036125183, 0.48331931233406067, 0.18518242239952087, -0.760219156742096, -0.6289088129997253, -0.7039162516593933, -0.5022252798080444, -0.025124158710241318, -0.07655595242977142, -0.11270968616008759, 1.1146355867385864, 0.2709970772266388, 0.02851259522140026, -0.8329260945320129, -0.5106940865516663, -1.267164945602417, 0.6335182189941406, -1.200356364250183, -0.8718937039375305, -0.8101345300674438, -0.8250539898872375, 0.08997367322444916, 2.066802740097046, 1.2440732717514038, -1.1657731533050537, 0.5155847668647766, 0.8381538391113281, -1.3921135663986206, 1.4612547159194946, -0.42679375410079956, 0.10024730116128922, -0.6513611674308777, -0.7298421859741211, 1.636698842048645, -0.6120082139968872, 0.2505994439125061, 1.418607234954834, 0.8427404165267944, -1.1439236402511597, 0.6753234267234802, 1.3130085468292236, 0.5668108463287354, -1.5574601888656616, -0.5902402997016907, -0.8601941466331482, 1.3390189409255981, 0.6863508820533752, 1.0284030437469482, -0.2220952957868576, -0.5869796872138977, 0.7128022313117981, 0.3956259787082672, -0.360588937997818, 1.3196024894714355, 1.1576546430587769, 1.0514919757843018, 0.8290863633155823, -0.972379207611084, 0.7611477971076965, 0.9491332769393921, 1.208801507949829, -0.4047795832157135, 0.2163800299167633, 0.7312610745429993, -0.9567924737930298, 0.8013842701911926, -0.10728254914283752, -1.9396306276321411, -0.7646788954734802, 0.11666738986968994, -1.118720293045044, 1.1914191246032715, -0.8245323896408081, 0.4819876551628113, -1.2512339353561401, 0.8404011726379395, 0.395770788192749, 1.3288594484329224, 0.17659993469715118, 0.44456687569618225, -1.0598872900009155, -0.018182868137955666, -0.8326951265335083, 0.9098207354545593, 0.0024923046585172415, 1.2831554412841797, 0.1695006936788559, 0.3576568067073822, 1.4015823602676392, 0.011893673799932003, 0.44510701298713684, -0.8919684886932373, 0.9872018098831177, -1.3359709978103638, -0.37986892461776733, 0.24070627987384796, -0.8596643209457397, -0.30170392990112305, 1.1857913732528687, 0.9680293798446655, 0.5076148509979248, 0.7071095705032349, -1.6503424644470215, -2.1665754318237305, -1.1109811067581177, 0.3244677186012268, 1.9029028415679932, 0.5243152379989624, 0.04747850447893143, -0.2642636001110077, -0.5885595083236694, 1.7550545930862427, 1.2426804304122925, 0.17064283788204193, 1.3661834001541138, 2.299969434738159, 0.6614236831665039, -1.5698366165161133, -0.11203081160783768, -0.6720382571220398, 0.9284823536872864, -0.6071557998657227, 0.5984262824058533, -1.0119754076004028, 0.8744437098503113, -0.30855274200439453, -0.055882204324007034, -0.03261568024754524, -0.0121092414483428, 1.0691933631896973, -0.7593540549278259, -1.1893559694290161, 1.1395230293273926, 0.7437875270843506, 0.26726391911506653, -1.1255666017532349, 0.4201812446117401, -1.164245843887329, 1.2608239650726318, -3.6402251720428467, 1.031295895576477, 0.9295044541358948, -0.14313647150993347, 0.3689213693141937, 1.1280301809310913, 0.23164790868759155, 0.802783727645874, 1.4922499656677246, -0.9444417953491211, -2.160832166671753, -1.0754345655441284, -0.06602101773023605, 1.390286922454834, 0.3918270766735077, -0.9362537264823914, -0.5586164593696594, -0.08819534629583359, 1.6501129865646362, 0.6536984443664551, -1.6150606870651245, -0.29360803961753845, 2.0040571689605713, 2.36860990524292, 0.8176719546318054, -1.058905005455017, -1.0927821397781372, 0.5518341064453125, 0.2218756526708603, 1.6882424354553223, -0.23101431131362915, 0.3442969024181366, -0.07236750423908234, 0.6093459129333496, -1.6260731220245361, 0.8323546051979065, 1.3378561735153198, -1.4674155712127686, 1.0934691429138184, -0.13795766234397888, -0.700444757938385, -0.2664382755756378, -0.46980196237564087, -0.9984145760536194, -0.4329608082771301, -1.546115756034851, -0.36593911051750183, -0.29447489976882935, 1.5619322061538696, 0.9569635391235352, 1.001359224319458, 0.9016309976577759, -0.4435553550720215, -0.4624038338661194, 0.8639348149299622, 0.19411775469779968, 0.14999040961265564, 0.75215744972229, 1.6147656440734863, -0.023404434323310852, 0.3575003445148468, -0.9881877899169922, 0.449294775724411, -0.7707750797271729, 0.06251093000173569, 0.948839008808136, -0.7963480949401855, -0.7073385119438171, 1.5663650035858154, 1.6468921899795532, -1.187281608581543, -0.39766183495521545, 0.6363674402236938, 1.0043176412582397, -0.7313752770423889, 1.6581401824951172, 1.3089532852172852, -0.40113916993141174, 1.6098721027374268, 0.34649062156677246, -0.6148188710212708, 0.4417034089565277, 0.6314570307731628, 0.004080727230757475, -0.4185035824775696, 0.26243653893470764, -0.6825262904167175, -1.2765969038009644, -0.3839634656906128, 1.4476807117462158, -1.6913951635360718, -1.055659294128418, -1.1135433912277222, -0.2729816734790802, -0.940271258354187, 0.3579883873462677, -1.6730570793151855, 0.020247474312782288, 1.96015465259552, -0.7747679948806763, 1.4068034887313843, 0.36197659373283386, 0.2558401823043823, 0.9628280997276306, 0.10613198578357697, 1.0123471021652222, 0.09467795491218567, -0.4208912253379822, 1.015762448310852, -1.0683197975158691, 0.7435844540596008, -2.334660530090332, -0.7476230263710022, 0.3190516233444214, -0.050976451486349106, -0.7278762459754944, 0.024604246020317078, 1.0256235599517822, -0.7271584868431091, -0.9237509369850159, 0.04171964153647423, 0.7784494757652283, -0.9778856635093689, -1.5583744049072266, -1.4296858310699463, -1.1033796072006226, 0.1543554812669754, -1.1881911754608154, -0.6297019124031067, -0.2396574467420578, 0.09396389871835709, 0.6684974431991577, -1.3667703866958618, 0.11768850684165955, -1.2076332569122314, -0.7064455151557922, -0.00965211633592844, -0.21325717866420746, 0.9683412313461304, 0.012013827450573444, -0.2617240846157074, -0.04084649682044983, 1.1807053089141846, -1.555830717086792, -0.8925859332084656, 2.1800661087036133, 0.3969898223876953, -1.6773836612701416, 0.7313064932823181, -1.4683371782302856, -0.5019984841346741, 1.8010655641555786, -0.870405912399292, 1.3621232509613037, -0.9459353685379028, 1.9896790981292725, -0.2045566290616989, 0.5223479866981506, -1.7160521745681763, 0.39359959959983826, 1.0589407682418823, 0.3477640450000763, 1.039589524269104, -1.2977614402770996, 0.8978445529937744, -1.624922275543213, 0.9767369627952576, 0.8933099508285522, -0.3604338765144348, -0.11743733286857605, -0.52474445104599, 0.07311414182186127, 0.4723600149154663, -0.4015708267688751, 0.23992405831813812, 0.8632890582084656, -0.9858332872390747, 1.0347694158554077, 0.8460948467254639, 3.9092326164245605, 2.523895025253296, 0.5721036791801453, -0.44648048281669617, 0.0649259090423584, 0.15063238143920898, 0.1270606368780136, 0.47806257009506226, -1.3740206956863403, -0.37454843521118164, 0.4860827922821045, 1.468720555305481, -0.3190191388130188, -1.314583659172058, -2.09458065032959, -0.9335184693336487, 1.312278151512146, -0.11217664927244186, -0.10308446735143661, 0.9816469550132751, 1.054194450378418, -0.4959741234779358, 1.0458887815475464, -1.38086998462677, 1.136013150215149, 0.4860965609550476, 1.0193727016448975, 1.778944492340088, 0.6189930438995361, -0.47489896416664124, -0.2220602035522461, -0.4828312397003174, 0.9830777645111084, -0.38797527551651, 0.07964968681335449, -1.2200639247894287, -0.06133836880326271, -0.6615358591079712, -0.3896290063858032, -0.34607434272766113, -1.036091923713684, 1.4688299894332886, 1.919792890548706, -1.3171710968017578, 0.12261434644460678, 0.5508716106414795, 0.566997230052948, -0.6044759750366211, -1.1725670099258423, 0.41711360216140747, -0.3820362389087677, 1.4394646883010864, -0.9373143315315247, 0.09562692791223526, -1.0913938283920288, 0.6057966947555542, 1.2522422075271606, 0.5899220108985901, 0.2667844891548157, -1.0814296007156372, -0.7590758204460144, -0.7855294346809387, 1.5282865762710571, 0.40894630551338196, 1.609872579574585, -0.006855509243905544, 0.644473671913147, -0.6211367845535278, -0.147133931517601, 0.027678392827510834, -3.9306087493896484, 0.4931112825870514, -0.8591502904891968, 0.345282644033432, -0.006612185388803482, -0.17798100411891937, 0.6968713998794556, 0.4001820981502533, 0.47640231251716614, -0.09375002235174179, -0.28283795714378357, 0.3126562237739563, -1.788184404373169, 1.0769392251968384, -1.228498935699463, 0.17389196157455444, 3.098432779312134, -2.003180503845215, 0.9624784588813782, 0.54088294506073, 0.07317198067903519, -0.5929851531982422, -1.4262845516204834, 0.640144944190979, -0.7593545317649841, -0.45410412549972534, -0.5774993300437927, -1.2139742374420166, 2.7396459579467773, -1.3958712816238403, 0.806876003742218, 0.48654648661613464, -1.8396852016448975, 0.1778067648410797, 0.6050459742546082, 0.6351522207260132, -2.147979259490967, 0.19096165895462036, -0.07399620115756989, 0.5077044367790222, 1.4500374794006348, -0.03223099932074547, 0.289837509393692, 0.10413360595703125, 1.7598768472671509, -2.4105019569396973, 0.11130601912736893, 0.08061103522777557, -0.4747968912124634, 1.9097896814346313, 0.15179535746574402, -1.2287734746932983, 1.007675290107727, -0.4957367479801178, -0.6393402218818665, -0.7683841586112976, 2.1236624717712402, 0.9397650361061096, 0.014952996745705605, -0.7777687311172485, 0.7446853518486023, 1.6311819553375244, -1.0005910396575928, -0.23959378898143768, -0.43747302889823914, 1.0028899908065796, 0.9584107398986816, 0.33516430854797363, -0.8756972551345825, 0.15284161269664764, -1.1496340036392212, 0.09227578341960907, 0.49614131450653076, 1.4174493551254272, -2.0289788246154785, 0.4161800146102905, 0.23035584390163422, -0.02811243012547493, 1.7391164302825928, -1.6929429769515991, 1.752090573310852, 0.8681224584579468, -1.1124783754348755, 2.629814624786377, 0.6710575819015503, 0.9146488308906555, 1.107692837715149, -0.17761333286762238, -0.6231001615524292, 0.015704471617937088, 0.6157755255699158, -0.06560301035642624, -0.9224528670310974, -0.5289134383201599, 0.17695198953151703, 1.342588186264038, 0.24935542047023773, 0.6895807981491089, -0.7151002883911133, 0.17769461870193481, -1.4649062156677246, 0.032988984137773514, -0.16189412772655487, -1.6784400939941406, -0.9547338485717773, 1.3733372688293457, -1.164326548576355, 0.8705400824546814, -1.5038268566131592, 0.9017375111579895, -0.5659160614013672, 0.7287359237670898, -0.10245861858129501, 1.0408118963241577, 0.4074394404888153, 0.6636988520622253, 0.19822712242603302, 0.34559816122055054, -0.04002801701426506, 0.7078425288200378, -1.0039777755737305, -0.5647023916244507, -0.7434071898460388, -0.1605333536863327, -0.273214727640152, -2.1777892112731934, 0.9849365949630737, 1.6193557977676392, 0.8511665463447571, -0.9784195423126221, -0.3233453631401062, 0.6595918536186218, 0.27330154180526733, -0.908200204372406, 1.598624587059021, -0.6460014581680298, 2.2357122898101807, 0.6677146553993225, -0.07276234030723572, 0.5297818779945374, -1.38973069190979, 1.6111302375793457, 0.22118474543094635, 2.1124725341796875, -0.8838428854942322, -0.05482688546180725, -0.33878475427627563, -0.03586495295166969, 1.2679225206375122, -0.9533559083938599, -0.6062042117118835, -1.6926705837249756, 0.12382294237613678, 0.14421318471431732, 0.740924596786499, 0.15499182045459747, -0.2917742133140564, 1.013607144355774, 0.9283236861228943, 0.3841688632965088, -1.060007929801941, 1.299185037612915, -0.3151661157608032, 1.2631263732910156, -1.2419289350509644, 0.06798865646123886, -0.5978664755821228, -0.4992399513721466, 1.049379587173462, 1.658966064453125, 1.035891056060791, -0.5157167911529541, 0.2396513819694519, 0.03616119548678398, 0.5334240794181824, 0.45125100016593933, 0.49203622341156006, -0.12041007727384567, -2.0488789081573486, -0.43924257159233093, -0.4364892244338989, -1.2527529001235962, 0.8527771830558777, 1.5873801708221436, 0.8106517791748047, -2.069157838821411, -1.3430033922195435, 1.771252989768982, -0.2842479944229126, -1.5040403604507446, 0.61957848072052, 1.4142236709594727, -1.763376235961914, 1.9257309436798096, -0.5757171511650085, -1.3519681692123413, 2.5526187419891357, 0.2709805369377136, -1.5113390684127808, -0.8166821599006653, 0.13586466014385223, -0.5645279288291931, -0.2527104318141937, 0.5489599108695984, 1.242104411125183, -1.205977439880371, 0.15833240747451782, 0.3234536647796631, 0.4964521527290344, -0.3198496401309967, 0.13475710153579712, 0.6618062257766724, -0.29775333404541016, 0.4159134328365326, -0.7788219451904297, 0.5763763785362244, -1.6932483911514282, -1.155566930770874, -0.3373953402042389, -0.40659064054489136, 0.3180379867553711, -0.6192349791526794, -0.09129667282104492, -0.2442770153284073, -0.4527902901172638, -0.16357094049453735, -0.4662218391895294, 0.6881336569786072, 0.6713782548904419, 0.6078296899795532, 0.09252368658781052, 0.2558712661266327, -0.14877811074256897, 0.545746922492981, -0.9094548225402832, -0.46039730310440063, 0.5392296314239502, 0.6582323312759399, -1.398975133895874, 0.24525707960128784, -0.17055274546146393, -0.23801285028457642, -0.7105793952941895, -1.4855948686599731, 1.3637495040893555, -0.43211594223976135, -1.6096267700195312, 0.8102754354476929, 0.9425650238990784, 0.046815987676382065, 0.12409069389104843, 0.6893110871315002, 0.4103788733482361, -0.3635200560092926, -1.0784640312194824, 1.2732994556427002, -0.047272246330976486, -0.449735552072525, 3.0671017169952393, 0.3071844279766083, 1.4296563863754272, 2.549116373062134, 0.21948915719985962, 0.030249785631895065, -0.703510582447052, -0.3132193982601166, -1.492224931716919, 1.0855532884597778, -0.16950289905071259, -0.09072238206863403, 2.314852476119995, 0.6428210735321045, -1.8652398586273193, -0.4987660348415375, 0.3113572895526886, 0.7509763836860657, -0.8399606347084045, -0.3500094711780548, -0.40241706371307373, -0.3050915598869324, -1.3568893671035767, -0.4260439872741699, 0.3619629144668579, 0.012522642500698566, -0.9856190085411072, 0.4104897975921631, 1.9900869131088257, 1.3071264028549194, 0.150248721241951, -3.2085511684417725, 0.3917429447174072, -0.7120289206504822, -0.38129881024360657, -0.3665727376937866, 0.29772496223449707, 1.8189773559570312, -0.8931195139884949, 0.8621624112129211, 1.9762684106826782, -0.5972903966903687, -0.5508185625076294, -0.7205400466918945, -0.6962588429450989, -0.8132312893867493, 2.3445510864257812, -0.37832111120224, 1.013386845588684, 0.6461286544799805, -0.11310408264398575, 1.1006566286087036, 0.21763169765472412, 0.0653114914894104, -0.46280157566070557, 1.213816523551941, -0.43633997440338135, 0.4010746479034424, 1.7341610193252563, 0.9965211749076843, -1.5199182033538818, -0.5074763298034668, -0.8433060646057129, 0.476655513048172, -0.09224759787321091, 1.1951700448989868, 1.1408172845840454, 0.6994743943214417, -0.27935564517974854, 1.2581839561462402, 1.2466533184051514, -1.2773411273956299, 0.3330681025981903, 1.0113656520843506, 1.0196393728256226, -0.8221413493156433, 1.1631968021392822, -0.05013217031955719, -1.2277888059616089, 2.1626296043395996, -1.1729323863983154, -1.7850263118743896, -0.2856966555118561, -0.48417332768440247, -0.7751377820968628, 1.8407881259918213, 0.6714252829551697, 0.7707633376121521, -0.11288994550704956, 1.2498078346252441, 0.9535865783691406, 2.198239803314209, -2.00303053855896, 0.05699189007282257, -0.20490683615207672, 0.017177589237689972, 0.5516560077667236], [1.1335996389389038, -1.4994430541992188, 1.31834077835083, 1.5724446773529053, 1.2221287488937378, -0.054486919194459915, -0.9545997977256775, -0.29262980818748474, -0.7001031041145325, 0.7835580110549927, 0.5331465601921082, 0.46687498688697815, -0.10072537511587143, -0.7912774682044983, 0.5944196581840515, 0.18296007812023163, 1.1036229133605957, 1.3974297046661377, -0.7011241912841797, 1.2265260219573975, 2.0414907932281494, 0.2891308665275574, 0.4255480170249939, -1.4682354927062988, -0.13347302377223969, 1.3918864727020264, -0.4784051477909088, 0.31459611654281616, -1.438582420349121, -0.09831706434488297, -0.8395463824272156, 0.37596550583839417, -0.3785316050052643, -0.6715115904808044, 0.013734587468206882, -0.5964454412460327, -1.2659295797348022, -0.020994758233428, 1.3106659650802612, 0.03678087890148163, 0.8545213341712952, 0.3410266935825348, -0.8546445369720459, 0.36449146270751953, 0.2426007241010666, 1.7479946613311768, -0.4253045320510864, 0.9297705292701721, 0.060837406665086746, 0.863054633140564, -0.05682812258601189, -0.35877034068107605, 0.4752265512943268, 1.5007349252700806, 0.3973082900047302, 0.37233397364616394, -1.4894144535064697, 0.02590152993798256, -0.7539474368095398, 1.3281828165054321, 1.136204481124878, 0.2276744842529297, 0.00045486781164072454, 2.0709176063537598, -0.5856180191040039, -1.1675972938537598, -0.30765220522880554, -1.1699763536453247, -1.1070257425308228, -0.1424613893032074, 0.6491901874542236, 0.1763269454240799, -0.20764105021953583, 0.7870901823043823, -0.5630530118942261, -0.2666216790676117, -0.7484409809112549, 0.3751823306083679, 1.3188921213150024, 0.6461536884307861, 0.4199993312358856, -0.4698505103588104, 0.05838979780673981, -0.36626487970352173, 1.0367740392684937, 0.8423449993133545, 1.2394658327102661, -1.3920012712478638, 1.268775224685669, -0.4154583811759949, 0.07062017917633057, 0.5698394179344177, 0.5447584986686707, 0.2661493122577667, 0.10913806408643723, -0.7468180656433105, -0.6865548491477966, -0.7906368374824524, -0.027803592383861542, -0.9365195631980896, -1.3317244052886963, 0.5956851840019226, -1.3021445274353027, -0.11044147610664368, -1.6127448081970215, 0.3625798523426056, -0.6865705847740173, 0.41262561082839966, -0.40006211400032043, 0.8659433126449585, 1.1542311906814575, 0.29476892948150635, -0.8765644431114197, -1.1060584783554077, -0.4669055640697479, -1.7135515213012695, -0.8228446245193481, -0.7323858141899109, 0.14894813299179077, 0.8793582320213318, 1.997402548789978, 0.5366663932800293, 1.0410939455032349, -1.1947485208511353, 1.4587037563323975, -0.651023268699646, -0.1015121266245842, 1.8428587913513184, 0.9631105065345764, 1.3781356811523438, 0.7011902928352356, -1.856230616569519, -0.6180500984191895, -0.18985792994499207, -1.430616021156311, 0.5674250721931458, -1.93669855594635, 0.720671534538269, -1.5928386449813843, -0.2517094910144806, 1.133292317390442, 0.4855024516582489, -1.579984188079834, 0.6285485029220581, -0.5158585906028748, -0.41409066319465637, 1.178045630455017, -1.5541431903839111, 0.4571637213230133, 1.229541540145874, -0.09848839789628983, -1.148192286491394, -1.2167614698410034, 0.07530400156974792, -0.21313393115997314, 0.6792353391647339, -0.5607782006263733, 0.2609798312187195, 0.48488613963127136, 1.9795185327529907, -0.6489323377609253, -1.5665966272354126, -0.08290591835975647, 0.673336386680603, 0.42659658193588257, -0.2948090732097626, -1.6412677764892578, 0.608697235584259, -2.258810043334961, -0.07322081178426743, -0.633185625076294, -0.31105437874794006, 0.3326184153556824, -0.9092538952827454, -1.912843108177185, -1.6599549055099487, -1.7938363552093506, -0.39830756187438965, 1.0033586025238037, 1.0731332302093506, 0.5113916397094727, -0.7346497178077698, -0.36910519003868103, 0.010751214809715748, 0.21426540613174438, -0.305624783039093, -1.3453021049499512, -0.7679352164268494, -0.942171573638916, -0.18850557506084442, -0.10260451585054398, 0.5202540159225464, 0.13146960735321045, 1.0858863592147827, 0.16765856742858887, 0.03257177397608757, 1.0830320119857788, -0.5376693606376648, 0.06109766662120819, -1.164031744003296, 0.5267021059989929, 1.037769079208374, -0.31966736912727356, 2.6542108058929443, -0.9818853139877319, -1.7262318134307861, -2.148160457611084, -0.5369868874549866, -0.00309759471565485, -0.9967179894447327, -1.9018962383270264, 1.9822574853897095, -0.39750227332115173, 1.324203372001648, -1.8157004117965698, -0.2995189130306244, -0.05368630960583687, -0.07041022181510925, 1.2134758234024048, 0.5943201184272766, 0.07025104016065598, 0.9152056574821472, -0.34830108284950256, 0.34197235107421875, 0.7954537272453308, -0.41771432757377625, 0.3567706346511841, 0.8098095059394836, 1.872846245765686, 0.5643717050552368, -0.6063869595527649, 1.4695965051651, -1.2072882652282715, 0.5696029663085938, -0.19520407915115356, -0.24592484533786774, 0.5382125973701477, -0.1631944477558136, -0.10452434420585632, 0.9480522871017456, -0.8423598408699036, 1.6471061706542969, -2.2436163425445557, -0.0786910131573677, -0.8961251378059387, 0.6141526103019714, 0.15738016366958618, -0.024927973747253418, 0.9029434323310852, 0.7312864065170288, -0.591273844242096, -0.055750176310539246, 0.14460574090480804, -1.1924885511398315, -0.5698213577270508, -0.11019475013017654, 0.08716608583927155, 0.15003702044487, 2.269092082977295, -0.8859216570854187, -0.2048298865556717, -0.06190329045057297, 1.4962732791900635, 0.428027480840683, -1.245831847190857, -0.20445583760738373, 0.5284866690635681, -1.624526858329773, -0.4761683940887451, -1.6454064846038818, 0.1002998948097229, 0.3921261131763458, -0.8293575644493103, -0.1933526247739792, -0.10513156652450562, 0.24677547812461853, 0.6239372491836548, 0.32694509625434875, -2.5509750843048096, 0.27177414298057556, -0.3868199288845062, 1.1111695766448975, -2.4202463626861572, -1.4879562854766846, -0.49952232837677, -0.6737939715385437, 0.5973621606826782, -0.014363082125782967, 0.8795269727706909, 0.6613752245903015, 0.6474431753158569, -0.32024717330932617, 0.10122548788785934, -0.4974510371685028, -0.4567870497703552, 1.0815218687057495, 0.4225543141365051, 0.7447094917297363, -1.0484960079193115, -0.5348474383354187, -2.53485369682312, 0.28644928336143494, -1.919998049736023, -0.2058713138103485, 2.194532871246338, 0.7161561846733093, 1.019993782043457, -0.16014327108860016, -0.1926785707473755, 1.4111659526824951, -0.7060620188713074, -0.8572380542755127, -0.6872400045394897, -1.8967547416687012, 0.6026170253753662, -0.8061240315437317, -0.21208472549915314, 0.37077638506889343, 0.6377553939819336, 0.4677521288394928, -0.12395232915878296, -0.2566904127597809, 0.15193404257297516, 2.1779825687408447, 0.5961064100265503, 1.3155317306518555, -0.913284957408905, -0.1973792463541031, -1.4999662637710571, -1.7248766422271729, 1.8493173122406006, -1.3821715116500854, 1.10322904586792, 0.4294756352901459, -1.4166052341461182, -0.6608110666275024, -1.465338945388794, -0.5358065366744995, 0.062269944697618484, 0.06104419752955437, -0.3193742036819458, 1.6317187547683716, -0.3922464847564697, -0.1080269142985344, -0.9284685254096985, 1.23770010471344, -0.3698386251926422, 0.6005224585533142, -0.9953038096427917, -0.2234935760498047, -0.4543996751308441, 0.4118396043777466, 1.5812691450119019, -0.5163302421569824, 0.8373246192932129, 0.7680453658103943, -0.4387800991535187, -0.9780388474464417, 0.8152716159820557, 0.8122157454490662, -0.06001048907637596, 0.025055767968297005, -0.05798327922821045, 0.6905062794685364, -1.3090399503707886, 0.6804729104042053, 0.49896639585494995, -1.2945972681045532, -0.3437688648700714, 0.8647617101669312, -0.01338404230773449, 1.878537893295288, 0.9301753640174866, -0.3397519588470459, -0.1384699046611786, -0.6569811701774597, 0.0594654381275177, -0.4624234437942505, 0.8597372770309448, -0.7114359140396118, -0.9199839234352112, -1.5522764921188354, -1.662163257598877, -0.11737179756164551, -0.07294915616512299, 1.1068724393844604, -1.2611757516860962, -1.1847624778747559, 1.3281834125518799, 0.6140323877334595, 2.3975720405578613, 0.8283481001853943, -1.9088577032089233, -0.624391496181488, -0.8919306397438049, 0.4571676552295685, 0.9648491740226746, 0.4757777154445648, -0.3207482397556305, -0.03964787349104881, -0.7712315917015076, 1.2652373313903809, 1.1860198974609375, 1.6435985565185547, -0.9489384293556213, -0.8202930092811584, 0.7843288779258728, -0.7763957977294922, -0.677442193031311, 0.39031609892845154, 0.6515209674835205, 2.6087846755981445, -2.5256705284118652, 0.7185866832733154, 1.6127976179122925, 0.8160764575004578, 1.2964746952056885, -0.3959352970123291, 1.9078258275985718, 0.021670924499630928, -0.07980514317750931, -0.5453488826751709, -0.19682469964027405, 0.11615808308124542, 1.4518547058105469, -0.434418648481369, 1.6126682758331299, 0.7127802968025208, 0.16861718893051147, -1.0844262838363647, 0.025835098698735237, -1.8419033288955688, -2.3168704509735107, 0.6180557608604431, -0.13963183760643005, -0.44911184906959534, 0.37253132462501526, 0.5055332779884338, -0.533424973487854, 0.9153475761413574, 1.1477566957473755, 1.9833399057388306, 0.31372225284576416, -1.6796845197677612, -0.5677464604377747, -0.10258692502975464, 0.37745919823646545, 0.12383215874433517, 0.5331621766090393, -0.3667190968990326, 0.9582757353782654, 0.3086875081062317, -0.5451325178146362, -0.15102258324623108, 0.18226784467697144, -0.883249819278717, 0.6037170886993408, 0.7630181312561035, -0.5709498524665833, 0.7504653334617615, 1.402832269668579, -0.00016234561917372048, 0.6117408275604248, -0.12686322629451752, -1.1277730464935303, -0.6424862742424011, -0.3187738358974457, -0.38360220193862915, -1.6856616735458374, 0.7906240820884705, 0.32038381695747375, 0.07018903642892838, 0.09067864716053009, 1.2433773279190063, 0.49147576093673706, 0.4830712676048279, -0.4053346514701843, 2.526226043701172, -0.24751710891723633, 0.9840530753135681, 0.4809842109680176, -1.7372119426727295, -1.033265233039856, -0.06056955084204674, 0.2933984398841858, 0.8384202718734741, -0.2228194624185562, -0.5779369473457336, -0.2230222523212433, -2.054084300994873, 0.8577725291252136, -1.0821669101715088, -1.597470760345459, 1.3616191148757935, 0.6077373027801514, -2.379230499267578, 1.1066347360610962, 0.48403769731521606, 0.4662109613418579, 1.1468437910079956, 1.1879944801330566, 0.5278751254081726, 0.1308608055114746, 0.3079052269458771, -0.3079845905303955, -0.4763706624507904, -1.2097532749176025, 1.1849390268325806, 0.47680243849754333, -0.8613232374191284, 0.07179617136716843, 0.4074094295501709, -2.3649468421936035, -0.6417834758758545, -0.43855351209640503, 1.4811646938323975, -0.9589824676513672, -0.2765093445777893, -0.26148292422294617, 0.08540349453687668, 2.261995315551758, -1.3441237211227417, -0.1948079913854599, -0.3703448176383972, 0.2975926697254181, -0.2793930172920227, -1.1306496858596802, 0.8541067838668823, -0.7297478914260864, -0.42471352219581604, 0.833768904209137, 0.10008231550455093, -1.0859124660491943, -3.091496706008911, -0.3630121350288391, 0.5805189609527588, 0.5723557472229004, -0.8408603072166443, -0.6268545985221863, 1.329383134841919, -0.15120340883731842, 1.1715463399887085, -0.8564057946205139, -2.4218685626983643, 1.5237891674041748, -0.45463183522224426, -0.44839516282081604, -2.3011832237243652, 0.048803601413965225, 0.5997629761695862, 1.3886743783950806, -0.9046079516410828, 0.0644688829779625, 0.4652627408504486, 1.4025357961654663, 0.23545104265213013, -0.7779061198234558, -0.1924552470445633, 1.2885037660598755, 1.9649579524993896, 0.19189581274986267, 0.22523260116577148, 0.7545618414878845, 0.524005115032196, 0.36697837710380554, -2.1471126079559326, 0.26720672845840454, 0.7982242703437805, 0.12047556787729263, -0.2748488485813141, -0.25222745537757874, -0.5616334080696106, 0.9729479551315308, 0.08718080818653107, -0.4421253800392151, -0.501886248588562, 0.24214637279510498, -1.2980003356933594, -1.467374563217163, 0.22435453534126282, -1.5187345743179321, -0.2799256443977356, -0.5880403518676758, -0.8417294025421143, -0.8194581270217896, -0.5522612929344177, 1.069096326828003, -1.1086227893829346, -0.6068188548088074, 2.176152467727661, 0.5347187519073486, -0.5003767609596252, -0.9356541633605957, 1.4628050327301025, -0.26132932305336, 1.4546159505844116, 0.8759598731994629, -0.09172599762678146, 0.6278376579284668, 0.00783631857484579, 0.7763187289237976, -1.0459253787994385, 0.9591094255447388, -1.5848562717437744, 0.3016815483570099, 0.45069271326065063, -0.6034421920776367, 0.3313734531402588, -2.393784523010254, 0.8656498789787292, 0.10768167674541473, 1.2142691612243652, 0.4915310740470886, 1.530861735343933, 3.4951319694519043, 0.10925329476594925, -1.4494084119796753, -0.1722828894853592, 0.31096866726875305, 0.6756072044372559, -1.8023215532302856, -1.1343491077423096, -0.05716331675648689, -1.6746753454208374, 0.5791348814964294, -0.8728926181793213, 0.2371675968170166, 0.16619043052196503, -1.4559690952301025, -2.3331401348114014, 1.058786392211914, 0.6458961963653564, -1.0333178043365479, -0.014171343296766281, 1.8082014322280884, -0.11471590399742126, 1.6443488597869873, 0.49688127636909485, 0.7251519560813904, 1.0309576988220215, 0.20744311809539795, -0.1077885702252388, -0.2819892466068268, 1.2865898609161377, -2.02238392829895, 0.3954981863498688, -0.951876163482666, 0.9095467329025269, 0.7297108173370361, 0.022759968414902687, 1.8671067953109741, 0.0005263472557999194, -0.3911411464214325, -2.380979537963867, 0.1660853922367096, 1.174056887626648, -1.0685269832611084, 0.3115718364715576, -0.5881990790367126, -2.1551060676574707, -0.38659676909446716, 0.565091073513031, -0.2955571115016937, 1.989644169807434, -0.6149089336395264, -0.17343923449516296, 0.15388086438179016, 0.48988500237464905, 1.743146300315857, -1.0119558572769165, -0.6503373980522156, 0.3462856113910675, 0.5750718116760254, -1.0319057703018188, -0.17817322909832, 1.1817282438278198, 0.20602452754974365, 0.7114683389663696, 0.03325919061899185, -0.7253842949867249, 0.8052932620048523, -0.38528120517730713, 0.04175860062241554, 0.2554663121700287, 1.9967142343521118, 1.4394609928131104, -2.0351552963256836, 0.044121935963630676, 0.5112507939338684, 0.6168502569198608, 1.207977056503296, 0.3173770606517792, -0.44365763664245605, 1.3833799362182617, 0.40077221393585205, 0.34190747141838074, -1.230002522468567, -0.23744694888591766, 1.495766043663025, -0.44497859477996826, 0.14680159091949463, 0.6801026463508606, 1.1580995321273804, 1.865069031715393, -1.870052456855774, 0.12125255167484283, -1.7600343227386475, -1.1541378498077393, -1.4482425451278687, 0.7461240291595459, -2.4952244758605957, 0.1668616384267807, -0.3154042363166809, -1.8784677982330322, 1.1773287057876587, 0.5359194278717041, -0.1412205696105957, 2.444063901901245, -0.7316853404045105, 0.9213029146194458, 0.7054293155670166, 1.3485556840896606, 0.11545193940401077, -0.8071768283843994, -0.6081219911575317, 0.34783482551574707, 1.4296302795410156, 0.5558691024780273, 1.3843715190887451, -0.6787204146385193, 0.8745866417884827, 1.4222055673599243, 0.14558818936347961, -0.03504399582743645, 0.46536993980407715, -0.9672097563743591, -0.6761376261711121, -0.11216443032026291, -1.3539167642593384, 1.0251860618591309, -0.15530039370059967, -0.6071003675460815, -1.2110446691513062, -0.25819599628448486, -0.10378570109605789, -1.8267651796340942, 0.21165144443511963, 1.2782694101333618, -0.11747580766677856, -1.5407954454421997, 0.9324235916137695, -0.19450363516807556, -0.3082777261734009, -1.6124216318130493, 1.3427785634994507, 0.4889313578605652, -1.0950396060943604, -0.33136722445487976, 1.2358440160751343, -0.5601378679275513, 0.24480418860912323, 0.7196755409240723, -0.03463927283883095, 0.7951019406318665, 1.0208430290222168, 0.04907756671309471, 0.4797680974006653, -0.07501788437366486, -0.11289253830909729, 0.5468035936355591, -0.6896167397499084, 0.446747362613678, -0.40753328800201416, -1.458755373954773, -1.0592869520187378, 0.5564377307891846, 1.1157498359680176, -0.16400666534900665, 0.7613702416419983, 1.5137462615966797, -2.476301908493042, -1.1501084566116333, 1.590360164642334, -2.1162164211273193, 0.4554959237575531, -0.9894936680793762, -0.2008494734764099, -1.3897606134414673, 1.1137573719024658, -0.978114664554596, -1.1374948024749756, -0.9637497663497925, 0.007929760962724686, 0.49279141426086426, 0.9305267333984375, 1.492258071899414, -1.6865437030792236, 0.7379547953605652, -0.32614126801490784, -1.5636543035507202, 0.053620871156454086, 0.15124787390232086, -0.23292426764965057, -0.32719337940216064, -0.09498348832130432, 1.0235657691955566, -0.11734228581190109, 0.7029291391372681, 0.629815399646759, -0.11910596489906311, -0.22146163880825043, 1.032395362854004, -2.2502596378326416, 0.023495396599173546, 0.15032018721103668, 0.44606056809425354, 0.19884908199310303, -0.3062339127063751, -0.4513470530509949, -2.005403757095337, 2.7522013187408447, 0.35270753502845764, 0.1378849893808365, -1.3975468873977661, -2.1072511672973633, 0.8449172377586365, 0.7283750176429749, -0.17889991402626038, 2.0154075622558594, -0.25828298926353455, -1.3246986865997314, 1.3599388599395752, -0.5910909175872803, 0.2361392378807068, -2.0905392169952393, 0.08746286481618881, 0.7094646692276001, 0.21326501667499542, 0.46481770277023315, 0.6487067937850952, -0.33965298533439636, -0.417917937040329, -0.5704113841056824, 1.328584909439087, 1.0650453567504883, -0.19240501523017883, 1.9461162090301514, -0.5208694934844971, -0.7199730277061462, 0.02672470733523369, -0.9530916213989258, 0.3691175580024719, -0.2864408493041992, 1.0209108591079712, 0.17290176451206207, 0.9765210747718811, -1.5341863632202148, -1.2882604598999023, -0.9269063472747803, 0.06474708020687103, 0.5352422595024109, -0.10048265755176544, 0.22088153660297394, -0.7672205567359924, 1.474557638168335, -0.9699189066886902, -0.21592764556407928, -1.9304813146591187, 1.1161539554595947, 1.837382435798645, -2.450204610824585, 0.5372133851051331, -0.2340088039636612, 0.4154241979122162, -0.1065487489104271, 0.20781077444553375, 1.9696471691131592, -0.791074812412262, -1.177341341972351, 0.14191386103630066, -0.8787250518798828, 0.03439220041036606, 0.031055543571710587, 1.1010533571243286, -0.46225860714912415, 0.41125234961509705, -0.8907450437545776, 0.8674607872962952, 0.18026356399059296, 0.2263069897890091, 0.1748971939086914, -0.7777796387672424, -0.4450663924217224, -1.3180371522903442, -0.04313215985894203, -0.34504956007003784, -0.6946452856063843, 0.4891728460788727, -0.17196078598499298, 1.2780849933624268, -1.2179228067398071, 1.2181862592697144, 1.9244321584701538, -0.5493218302726746, -0.18000055849552155, -0.29975056648254395, 0.04147244989871979, -0.22033141553401947, 0.16256675124168396, 0.15228748321533203, 0.006284909322857857, -1.171400785446167, -1.2951207160949707, -0.14075149595737457, -1.246045470237732, 0.8491272926330566, 0.8850002288818359, -0.5758060812950134, -1.1109416484832764, 1.2715131044387817, 1.305928349494934, -1.4712523221969604, -0.8284781575202942, -0.6577087640762329, -0.7594562768936157, 1.4350049495697021, 0.04042569920420647, 0.1544661670923233, -2.27036452293396, -1.5257086753845215, -0.666592001914978, 0.41716521978378296, 0.5586851239204407, 0.8450068235397339, -1.9387083053588867, -1.0098565816879272, -0.35749351978302, -2.0058181285858154, -0.0641237273812294, 1.6417665481567383, 0.1859363466501236, -0.33861687779426575, 0.5670642256736755, 0.6667897701263428, 0.6627439856529236, -0.5702724456787109, -0.11372366547584534, -1.406829595565796, -2.313884973526001, -0.029420368373394012, 0.6184273958206177, 0.060947537422180176, 0.9239938259124756, 0.03661240637302399, -0.9581596851348877, 0.14503110945224762, 0.3254002034664154, -1.4365355968475342, 0.825760006904602, 1.3837937116622925, 0.8676685690879822, -2.303816318511963, -1.227292537689209, 0.8071736693382263, 1.2349276542663574, 0.12967655062675476, 0.6994302868843079, 1.067457675933838, -0.8782504200935364, -1.2722523212432861, 0.517261803150177, 0.8486798405647278, 0.5176025629043579, 0.2578940987586975, -0.5779559016227722, 0.15343506634235382, 0.5468177199363708, 0.38341212272644043, -0.9236688613891602, 0.5820384621620178, -0.21830207109451294, -0.04961836338043213, -0.1971217840909958, -1.2634199857711792, -0.43297678232192993, -0.8921162486076355, 0.8667492270469666, 0.8280184268951416, 0.14113126695156097, -0.7258213758468628, -0.5430657267570496, -0.2634032964706421, -0.2229757308959961, -0.3910623788833618, 0.530228853225708, 1.190323829650879, -0.509161651134491, 0.469953328371048, -0.1296314150094986, -0.4713919758796692, -0.7296977639198303, 1.2049388885498047, 0.5788933038711548, -1.2761472463607788, -0.3747903108596802, 0.47599855065345764, 0.8595077395439148, 0.6025338172912598, 0.4570101797580719, -0.15276633203029633, 0.003806701395660639, 0.18739275634288788, 1.6631277799606323, 1.1027841567993164, -0.8100102543830872, -3.0625576972961426, 0.49988046288490295, 1.566903829574585, -0.047079864889383316, 0.36822402477264404, -1.4181950092315674, 1.1748723983764648, -1.029122233390808, -1.6478115320205688], [-0.43323495984077454, 0.7247700095176697, 0.36404940485954285, 1.1204389333724976, 0.5535153746604919, 0.5007014870643616, -0.5421570539474487, -2.5732953548431396, -0.13424181938171387, -1.1447654962539673, 0.8242541551589966, -0.5772956609725952, 0.4807933568954468, 0.3043864071369171, 0.10253161191940308, 0.7963100075721741, 0.6027020215988159, -0.7619196176528931, 0.7751174569129944, 0.5181031227111816, 0.4726078510284424, -0.9462028741836548, 0.2731437385082245, -0.9556629657745361, -0.8958810567855835, 2.099682569503784, 0.40386003255844116, 0.02224535495042801, 0.4446895122528076, -0.90473872423172, 1.2592440843582153, 1.3799636363983154, -0.8517138957977295, -0.5191400647163391, -0.47667166590690613, -0.019923493266105652, -0.47352156043052673, -1.889949917793274, 0.2384946346282959, -0.2987067401409149, 0.9581601023674011, -1.775963544845581, 1.5315171480178833, -0.03444435074925423, -0.6617552042007446, 0.9284579753875732, -0.492798388004303, -0.8317574262619019, 0.733173668384552, 0.6878185272216797, 0.3180355429649353, 0.5958253741264343, 0.7204930186271667, 0.10279430449008942, -0.9442587494850159, 0.3338926136493683, 1.1723204851150513, 0.3096882402896881, 1.4272252321243286, 1.3993308544158936, -0.21432332694530487, -1.0651984214782715, 0.14723677933216095, 0.9726362824440002, 0.5613749027252197, -1.1306829452514648, 0.38252559304237366, -0.280373215675354, 0.7781729102134705, -0.8158328533172607, -1.422664999961853, -0.6270207166671753, 0.7780705690383911, 0.848190188407898, -0.724187970161438, 0.3533402979373932, -0.08132899552583694, 0.4868304133415222, 0.12523393332958221, -0.05932213366031647, 1.851714849472046, -0.4124602675437927, -1.544852614402771, -0.4892185628414154, 1.7767201662063599, -0.6197655200958252, -0.1135835126042366, 0.8133634924888611, 0.6281828880310059, 0.015252628363668919, 1.2539150714874268, -0.18693695962429047, 0.74128258228302, -0.047933533787727356, -1.9883008003234863, -2.034816026687622, -1.2628275156021118, 2.1060540676116943, -0.06892300397157669, -1.5609859228134155, 0.31541121006011963, 0.2690499424934387, -0.2949894666671753, -0.9900559186935425, 0.42374953627586365, -0.7637526988983154, -0.6902154088020325, -0.8097625970840454, 0.3615400195121765, -0.286153107881546, -1.6082921028137207, -0.15154679119586945, 1.7388408184051514, 0.7501015067100525, 0.07857965677976608, 0.032628901302814484, -1.6096720695495605, 0.8244410753250122, 1.2940350770950317, -1.2789109945297241, -0.4396044909954071, 0.08109429478645325, -0.7344179749488831, -0.344777375459671, -1.0678458213806152, -0.9887048006057739, -0.551673412322998, 1.431532859802246, 0.12257774919271469, 1.7819151878356934, 0.7444334626197815, 2.191826581954956, -0.060957834124565125, -1.0361168384552002, -1.639859676361084, 0.8923695683479309, -0.7913670539855957, -0.7637850046157837, 0.0642857700586319, -0.470062255859375, -0.17730656266212463, -0.9453766345977783, 0.0021468535996973515, 0.05271647125482559, 0.10476644337177277, -0.08368887007236481, 0.058645568788051605, -1.5403225421905518, -1.0529670715332031, -1.4366430044174194, 0.49296972155570984, -0.5094485282897949, 0.17989253997802734, -1.8601832389831543, -0.02751927822828293, -0.49927759170532227, -0.4441792070865631, -1.8348972797393799, 0.531798243522644, 0.7496182322502136, 0.2674046456813812, 0.3331672251224518, -0.3740987479686737, 1.9363696575164795, -0.06902783364057541, -0.2953459918498993, 0.01910433918237686, -0.05285916477441788, 0.44962823390960693, -1.0457115173339844, -0.14090484380722046, -1.8567583560943604, 1.0963107347488403, 0.6160100698471069, -2.4246671199798584, -1.4271451234817505, 1.2399873733520508, 0.5375626087188721, -0.17316125333309174, -1.6179378032684326, 2.9847662448883057, -0.5610156059265137, 0.9399852156639099, 0.3732060194015503, -0.7693013548851013, -1.066896677017212, -0.5691236257553101, 1.7227678298950195, 0.05962323024868965, -0.5119791626930237, -1.008624792098999, 0.6228524446487427, 0.5619543194770813, -0.7755201458930969, 1.242449164390564, -0.48142918944358826, 0.6682815551757812, 0.634748101234436, 1.7781791687011719, 0.09212508797645569, -0.2486019879579544, 0.1254720836877823, -0.09062950313091278, 0.32088860869407654, 0.7626354694366455, -1.6306205987930298, 0.6059800982475281, 0.7075738906860352, -0.32528236508369446, -0.3168969750404358, -0.9821285605430603, 0.13682334125041962, 1.6964575052261353, -1.3797484636306763, 0.6532427072525024, -1.655139446258545, -0.275170236825943, -0.36250123381614685, -0.48720163106918335, 0.8013086318969727, 2.0240590572357178, 0.10769526660442352, 0.3671039938926697, 0.7784249782562256, 1.1401598453521729, 0.08062171190977097, -0.1259343922138214, -0.705453634262085, -0.3764403462409973, 0.5891175866127014, 0.5115417242050171, -0.5524292588233948, -0.5707491040229797, 0.44661131501197815, -0.39809465408325195, 1.2920061349868774, 1.0401495695114136, -1.0167375802993774, 1.1863441467285156, -0.9817761182785034, -1.2353330850601196, 1.7728424072265625, -0.9467582106590271, 0.6316708922386169, 0.2951258718967438, -0.44442734122276306, -0.09530167281627655, 0.3360244333744049, -1.0223424434661865, -0.3004966974258423, -1.6974878311157227, -0.3752053678035736, 0.9109882712364197, -1.0190805196762085, 0.1859888732433319, -1.889939785003662, -0.5416474938392639, 0.07330893725156784, -0.8528884053230286, 1.2399663925170898, -2.1982004642486572, -0.5747355818748474, 0.9225165247917175, 1.837174892425537, -0.03922382742166519, -0.7339866757392883, -1.7118799686431885, -0.15079949796199799, -0.7692880034446716, 1.0786024332046509, -0.3038268983364105, -0.8113278150558472, -0.0807395651936531, 1.2299925088882446, 1.6721680164337158, 0.6451281309127808, -1.4997221231460571, -0.055496443063020706, -0.6491231918334961, 1.5202181339263916, 0.07414638996124268, 0.05191831663250923, 1.5678859949111938, 1.1519205570220947, -0.43869826197624207, -1.1827856302261353, -0.7837540507316589, -1.7482894659042358, -0.5338210463523865, 0.33217865228652954, -2.1288676261901855, 1.018587350845337, 1.5032017230987549, 1.6323379278182983, -0.9761114120483398, 0.8880871534347534, 0.1939896047115326, 0.5522236824035645, -0.31601187586784363, 0.24787573516368866, 1.246328353881836, 0.12992219626903534, -0.01284247636795044, 0.3421819806098938, 0.5948569774627686, 2.344702959060669, -1.1254643201828003, -0.18099357187747955, -1.2377715110778809, -0.0012925644405186176, -1.0209287405014038, 0.44433197379112244, 0.09147249162197113, 0.9112424254417419, -0.19441013038158417, -0.07384281605482101, 0.5396896600723267, -0.4656197428703308, 0.6506109237670898, -3.7543814182281494, -0.9518820643424988, 0.6121588945388794, -0.5713698267936707, -0.14027665555477142, -0.7646793723106384, -1.4161173105239868, -0.6906885504722595, 0.30813881754875183, -0.6955787539482117, -0.35577306151390076, 1.1272975206375122, -0.5352098941802979, -0.24665239453315735, 0.2527112662792206, 0.8407530784606934, -0.06021486222743988, -0.46890342235565186, 0.5305355787277222, 0.7053943872451782, -0.06763016432523727, -0.27199432253837585, 0.14988380670547485, -0.16931703686714172, 2.1388471126556396, -0.19928735494613647, -0.7659379243850708, -0.05253053829073906, 0.4301537871360779, 0.5877063274383545, -1.1687434911727905, 1.6315821409225464, -0.6032955646514893, 0.4993934631347656, 2.1542203426361084, 0.8638367652893066, -1.2422261238098145, -1.152249813079834, -0.06546251475811005, 0.5352131128311157, 0.8333041667938232, -0.11991672962903976, -1.652903437614441, 0.05296562612056732, 0.2934725880622864, 0.378269761800766, 0.4855324327945709, -1.5323795080184937, 0.8933265209197998, 0.32613605260849, 1.5609341859817505, -1.682335615158081, 1.0593065023422241, 1.9186127185821533, 0.46320778131484985, -0.44702598452568054, -0.2719442546367645, -1.0577433109283447, -2.0181448459625244, 1.624761700630188, -0.2433508038520813, -0.52689528465271, -0.7465490102767944, 0.37538883090019226, 0.6793915033340454, -0.08993778377771378, 1.2991468906402588, 0.1060306653380394, 1.141211986541748, -2.1569247245788574, 0.5684938430786133, 0.6027758121490479, 0.3353101909160614, 1.5501097440719604, 0.7276685237884521, 0.1061052680015564, -0.09870502352714539, -2.517319917678833, 0.749086856842041, -1.0509841442108154, -0.007097374182194471, 0.7765763998031616, -0.6791884899139404, -0.7793000936508179, 0.8436768054962158, -2.2228569984436035, -0.34690648317337036, 1.330594539642334, 0.06374672055244446, -0.9304337501525879, -0.6580144762992859, 0.05701524764299393, -1.385251760482788, -0.15227968990802765, 0.5643833875656128, -0.28165462613105774, 0.5524200797080994, 0.12218007445335388, 0.39818274974823, -0.6353992223739624, -0.6540265083312988, 0.04756538197398186, -1.3073606491088867, 1.3987282514572144, 0.9836748242378235, -0.2734668254852295, -0.6327654719352722, 0.8975115418434143, 1.2633622884750366, -0.25273582339286804, 0.45216020941734314, -1.259281039237976, -0.3339430093765259, 0.8270068764686584, -1.6565067768096924, -0.4448695778846741, -0.05490441620349884, -0.559268593788147, 1.4996085166931152, 1.312919020652771, -1.6247478723526, 1.0848779678344727, 1.1348050832748413, -0.5863504409790039, -0.21031050384044647, 1.1493182182312012, -0.6435750126838684, -3.168186664581299, 0.08390139043331146, 0.6080629825592041, 0.6007209420204163, 0.1832779347896576, 1.442942500114441, 0.6064163446426392, 0.9688997268676758, -0.32712191343307495, 0.15706515312194824, 0.3189947009086609, 0.42670759558677673, -2.4253711700439453, 0.6811031103134155, -0.1596319079399109, -0.7039945125579834, -1.0049880743026733, 0.29655811190605164, -0.653502881526947, 0.16573160886764526, -1.5403152704238892, 1.195008397102356, -0.702419102191925, -0.002869092160835862, -0.30285465717315674, 0.9743320345878601, 0.07121482491493225, -1.2635908126831055, 0.08811217546463013, 1.3871499300003052, -0.6562482714653015, 0.7876446843147278, -0.8324958682060242, -1.7165493965148926, 0.8928962349891663, 1.6552581787109375, -0.8258700966835022, 0.21321965754032135, 0.5029135346412659, 0.10975716263055801, -0.9006375074386597, 0.0969499871134758, -1.0628865957260132, 2.2358500957489014, 0.756482720375061, -0.6504045724868774, -0.41220614314079285, 0.6116490364074707, 1.5889222621917725, 2.1751770973205566, -1.368694543838501, 0.6970740556716919, 0.9862602353096008, 0.3937711715698242, -1.3243408203125, 0.3230256736278534, -0.6368934512138367, -0.3595898747444153, 1.2322731018066406, 0.7783006429672241, 0.7696139812469482, -1.241004467010498, -0.5224652886390686, 1.0045424699783325, 0.15509213507175446, 1.1365468502044678, -0.8443397879600525, 0.7930927872657776, -0.37045690417289734, 0.16155947744846344, 0.31755292415618896, -0.18374381959438324, -0.26338648796081543, -1.5433181524276733, -1.5516722202301025, -0.2842530608177185, 0.7430628538131714, -0.27332839369773865, -0.7540629506111145, 0.6673817038536072, -0.7124338746070862, -0.5491243004798889, -0.6274422407150269, 0.04900344833731651, 0.7864775061607361, -0.6974640488624573, -0.011486238799989223, -0.14012445509433746, 0.6121293306350708, -0.596919059753418, -0.5359457731246948, -0.15915752947330475, -0.3488601744174957, -1.0774123668670654, -2.5452022552490234, 0.32812580466270447, -1.3587443828582764, 0.7200119495391846, -0.027643373236060143, -1.4712404012680054, -0.8171638250350952, -0.808351993560791, 1.0048282146453857, -0.6392690539360046, -1.4430689811706543, -0.37748366594314575, 0.11897926777601242, 0.027549484744668007, -0.762145459651947, 1.9248621463775635, -0.9082278609275818, 1.8025360107421875, 0.4268718957901001, 1.0110663175582886, 0.618537425994873, -1.3852665424346924, -1.1178956031799316, -0.07508789747953415, -0.7568029761314392, 0.5455735325813293, -0.7356398105621338, -1.1310049295425415, -1.8409507274627686, 0.6450673341751099, -0.31913796067237854, 1.2050970792770386, 0.05436313897371292, 1.3019351959228516, 0.8055515885353088, -0.9648270010948181, 2.042485237121582, 0.4810130000114441, 1.9699978828430176, -0.23637442290782928, 0.24380375444889069, 1.4534378051757812, 0.32092130184173584, -0.7770086526870728, -0.9067955017089844, 0.27311110496520996, -0.3615407943725586, 0.4341031312942505, 0.8437647223472595, -0.610371470451355, -0.6381924748420715, -0.6232646703720093, -0.5197742581367493, 0.059198684990406036, 0.1769580990076065, -0.6419546604156494, -0.8081255555152893, -0.8063598871231079, -0.6131104230880737, 0.3982721269130707, -0.6320627927780151, 2.350710391998291, -0.6023643612861633, -0.3178473711013794, 0.32736337184906006, -0.8488046526908875, -0.8158789277076721, 0.965063750743866, 0.1769828349351883, -0.7985599040985107, 0.9336750507354736, 0.09275256842374802, -0.7052550911903381, 0.32214877009391785, 0.012385373003780842, -0.26173412799835205, -0.6204174160957336, 0.29511234164237976, -0.7321732044219971, -0.14818516373634338, -2.2574238777160645, -0.744817852973938, -1.695398211479187, -0.47031089663505554, -1.3716301918029785, -0.4984714984893799, -0.8476308584213257, 0.05059085786342621, 1.0594699382781982, -1.4115431308746338, 0.5081241726875305, -0.1513679027557373, 0.1653449386358261, 0.0990159809589386, 1.2547451257705688, -0.6380021572113037, 0.017434289678931236, -1.0439401865005493, 0.33360543847084045, 0.5125119090080261, 0.2786475718021393, -0.505982518196106, -0.05478654056787491, 1.055491328239441, -1.875765085220337, 1.6074433326721191, 0.062375616282224655, -1.0920464992523193, 0.8441768884658813, 0.3090498149394989, 0.31027889251708984, 0.13899341225624084, 0.40486329793930054, 1.261403203010559, 1.8905389308929443, -0.6487720012664795, -0.8504303097724915, 0.045047834515571594, 0.7095913290977478, -0.13824602961540222, -0.3458201587200165, -0.12155736237764359, -1.6152007579803467, 0.6049023270606995, 1.1987420320510864, 0.1352207511663437, 1.2017847299575806, -1.40938138961792, 1.5307469367980957, -0.9115014672279358, -0.14748616516590118, -1.1906999349594116, -1.3425945043563843, -1.3343520164489746, -0.303798109292984, 1.780455470085144, -0.6516737341880798, -1.12547767162323, 0.8808491230010986, -1.4813562631607056, 1.2554011344909668, -0.9099264740943909, -1.2401628494262695, 0.44724583625793457, -1.9375801086425781, -1.9418014287948608, 1.9876524209976196, -0.4586719870567322, 0.027822544798254967, -1.551188349723816, 0.21896883845329285, -0.20029734075069427, 0.3703092634677887, 0.335877925157547, 0.4286591410636902, -0.45783132314682007, -0.6381179094314575, -0.659150242805481, 0.4958917498588562, 0.9041222333908081, -0.7792750597000122, 0.3597157597541809, -0.15823204815387726, 0.27164211869239807, -0.9208104610443115, 1.8945519924163818, -0.461931973695755, 1.1040462255477905, -0.03741663321852684, 0.7362687587738037, -1.230954647064209, -0.9514127969741821, 0.44966813921928406, 0.6046620011329651, -0.4428776204586029, -0.18586979806423187, -0.43240851163864136, -0.8424316048622131, 0.09313684701919556, -1.4691301584243774, -1.477296233177185, -1.6072266101837158, 1.2716748714447021, -0.04081860929727554, -0.696978747844696, -0.03708425536751747, 0.5514959096908569, 1.593940019607544, -0.5911176204681396, 1.0368655920028687, -0.5651987791061401, 2.363827705383301, 0.9151495099067688, 0.4732632040977478, -0.05797569453716278, 0.48740530014038086, 1.2997478246688843, 0.6727508902549744, -0.11381755024194717, 0.4037056565284729, -1.124258279800415, 0.5633553862571716, -0.8298012018203735, 1.559693455696106, 0.16618458926677704, 1.897807240486145, 0.20005977153778076, -0.506500780582428, -0.5923733711242676, 0.47914475202560425, -0.2497369349002838, -0.45627206563949585, 0.31203967332839966, -0.9658394455909729, 0.4922035336494446, 1.0534623861312866, 0.6974354386329651, -0.31727179884910583, -0.0884007066488266, -0.7476978302001953, -0.4265863001346588, -0.7200031280517578, 0.0158647820353508, -0.007952575571835041, 2.3984906673431396, -1.738478183746338, 0.1722354143857956, 1.086833119392395, -0.49051621556282043, -0.20973561704158783, 0.7261972427368164, 0.0064702341333031654, -0.39309802651405334, -1.7521600723266602, -0.3810153603553772, -0.6533311009407043, 0.4841373562812805, 0.3455413579940796, 0.6682127118110657, 0.20711766183376312, -1.652690052986145, -0.7007869482040405, 0.215223029255867, -1.7399659156799316, -0.46912115812301636, 1.1633363962173462, -1.4718551635742188, 0.09211886674165726, 0.15804216265678406, -0.7816236019134521, 0.8584340214729309, 0.9383583664894104, -1.035080075263977, -0.8117724657058716, -0.9763227105140686, 1.2046277523040771, -0.6316990256309509, -0.3502490222454071, 0.7753551602363586, 1.5000391006469727, 1.9589875936508179, 0.7205469608306885, 1.1532715559005737, -1.778709888458252, 1.0168596506118774, -1.6899220943450928, -1.6103410720825195, -0.38903331756591797, 0.20057697594165802, 0.9310473203659058, 0.1910238265991211, -1.3631479740142822, -0.7379288077354431, -0.0009197022300213575, 1.1289258003234863, -0.6709463000297546, -0.29464414715766907, 0.5807377099990845, 0.27034229040145874, 1.7849271297454834, -0.5911624431610107, 1.405800223350525, 1.594208002090454, -1.8352612257003784, 1.99874746799469, -1.4383931159973145, 0.7456499934196472, 0.16773734986782074, -0.12850044667720795, -1.6919792890548706, 0.02708064392209053, -0.4159623980522156, -0.9239876866340637, -0.8557649254798889, -1.4261823892593384, 2.910257339477539, -1.3965646028518677, -1.5618690252304077, 0.14347957074642181, 0.42669913172721863, -1.222753643989563, -0.5551127791404724, -0.9961122274398804, -0.533414363861084, 0.8456606864929199, -0.29252195358276367, 0.09660021960735321, 0.172329381108284, 0.19198353588581085, -1.9242135286331177, 0.8125377893447876, 0.7255334854125977, -0.5578312873840332, 0.9013996720314026, -0.9731555581092834, -0.006566079333424568, -0.21105177700519562, 0.25924113392829895, 0.10078743100166321, -0.569901704788208, 1.5399060249328613, 2.2217042446136475, 0.26079967617988586, 1.1957933902740479, 0.171031191945076, 0.029932554811239243, -2.2332820892333984, -0.41649341583251953, 0.001480830367654562, -1.8709877729415894, 0.46027517318725586, -1.4213144779205322, 0.9012977480888367, 0.3552124500274658, -0.2891642153263092, 0.08867247402667999, 0.3856702148914337, 0.05906205624341965, 0.4346729815006256, 0.9957187175750732, 0.737209141254425, -1.0697954893112183, -0.751630961894989, -0.6843867897987366, -0.46194106340408325, 0.12078723311424255, -0.7553804516792297, 1.8921607732772827, -0.6073755025863647, -2.360098123550415, -0.7484833002090454, 1.1973912715911865, 0.46370232105255127, -1.0645300149917603, -0.2725064754486084, 0.44555166363716125, 0.9543373584747314, -0.7259286046028137, 1.6265558004379272, -0.03516894578933716, 1.2634694576263428, 0.8352903723716736, -0.44655442237854004, -0.19635745882987976, -2.2013978958129883, 2.3359692096710205, -1.6285104751586914, 0.12403491884469986, 0.37566080689430237, 1.351375699043274, -1.9347881078720093, 1.210748553276062, 0.6474708318710327, -0.30406418442726135, -0.26933473348617554, 0.5014499425888062, -1.6698004007339478, 2.7196571826934814, 0.3043682277202606, -0.7913596630096436, -0.7687079310417175, -0.06391991674900055, 0.5133083462715149, -1.9521474838256836, 0.932647705078125, -1.3452720642089844, -1.5584732294082642, 0.6837039589881897, -0.6292591094970703, 1.7116496562957764, -1.1685831546783447, -0.31488364934921265, 0.9184489846229553, -0.4447914958000183, 0.5770177841186523, -2.2652604579925537, 0.11395170539617538, -1.1740798950195312, 0.3743215799331665, -1.4001953601837158, 1.003888726234436, -0.5958195328712463, -1.9948384761810303, 0.09545735269784927, -1.1299313306808472, -0.8765217065811157, -0.24481910467147827, 0.2997613549232483, -0.6943625807762146, -0.04183845594525337, -0.32741880416870117, -1.212500810623169, 0.21754984557628632, 0.10824369639158249, 1.31289541721344, -1.0606638193130493, 0.32177290320396423, -0.7439916133880615, -0.11058195680379868, 0.7374981045722961, -0.17551913857460022, 0.07905826717615128, 0.11494895070791245, -1.152915596961975, -0.3750230371952057, 0.4578016996383667, 0.40477046370506287, -0.7244463562965393, -0.4219977855682373, -0.07844556123018265, -0.7161579728126526, -0.09894630312919617, -1.0836491584777832, -1.8614240884780884, 0.028245670720934868, -2.1167495250701904, -1.736111044883728, 1.110224962234497, -1.3141307830810547, 1.522286295890808, 0.8379905819892883, 0.8643912672996521, -1.3559257984161377, 0.561317503452301, 0.48186036944389343, -0.5889055132865906, 1.3816332817077637, 0.9003922939300537, -0.03499753773212433, 2.264573574066162, 1.0918620824813843, -2.2117130756378174, -0.45969411730766296, 1.4889887571334839, -2.3549795150756836, 1.093511700630188, 0.943549394607544, 1.0286835432052612, 0.43562859296798706, -0.418923944234848, -0.23594267666339874, 2.1116013526916504, 0.7032771110534668, 0.07946889102458954, -0.7707268595695496, 1.5233149528503418, -0.5999433994293213, 0.6991016864776611, 1.2134259939193726, 0.07660143077373505, -0.4843441843986511, 0.9989208579063416, 0.384965717792511, 0.4751976728439331, -0.22194018959999084, -0.07382487505674362, 0.6584720015525818, 1.5467811822891235, 0.5618681311607361, -0.7870744466781616, 0.475145161151886, -1.2733900547027588, 0.5310353636741638, 1.3345893621444702], [-0.8144726157188416, 0.7296180129051208, -0.8555734157562256, 1.656345009803772, -1.7901320457458496, 1.4438024759292603, 1.411454200744629, -0.3274769186973572, 0.6265566945075989, 1.6378581523895264, 0.6933188438415527, 0.4353677034378052, -1.0235087871551514, -0.21066968142986298, -0.07364118099212646, -0.2087848037481308, -0.3459494709968567, 2.0935964584350586, -0.34314730763435364, 0.010962062515318394, 0.07406085729598999, 0.5477846264839172, -0.9064316749572754, 0.9331001043319702, 2.5730056762695312, -0.2598058879375458, -1.3550666570663452, 0.05429327115416527, -0.9485054016113281, -0.7394181489944458, -0.2162303775548935, -0.13359487056732178, -1.1929235458374023, 0.9054388999938965, 1.194894790649414, -0.6459452509880066, 0.5877711772918701, 0.4708735942840576, -0.23201560974121094, 0.553092896938324, 0.006341157015413046, 0.9455534815788269, -0.047680579125881195, 0.13401269912719727, 0.8202371001243591, 1.3313133716583252, 0.18617938458919525, 0.029347511008381844, 0.6648570895195007, 1.8383723497390747, -0.6239942908287048, 0.012990228831768036, 2.507920980453491, -1.695509433746338, 0.5619620084762573, -0.13663235306739807, -1.091766357421875, -1.8512721061706543, -2.570042610168457, 1.720024585723877, 0.5888586640357971, -0.954466700553894, -1.4200949668884277, 0.1891992837190628, -0.021333133801817894, -0.22098101675510406, 1.653252124786377, -0.7818266153335571, -1.1697748899459839, 0.5882471203804016, -0.49131497740745544, -2.5037529468536377, -1.491999864578247, -0.3115139305591583, -0.6377907395362854, -1.0513575077056885, 0.24679094552993774, 1.026543140411377, 1.2693970203399658, 0.6662507057189941, 0.6134582161903381, -0.02496497891843319, 0.8045437335968018, -0.05030645802617073, 1.4042607545852661, -1.1027361154556274, -0.9106002449989319, 0.14456945657730103, 0.9006274342536926, 0.6450399160385132, 0.36298462748527527, 1.2885531187057495, -0.33308088779449463, -0.15908683836460114, 1.1316277980804443, -1.1987555027008057, -3.0601654052734375, 2.43330717086792, -1.5273009538650513, -0.5354732871055603, 1.2753335237503052, 0.6656643152236938, 0.6112292408943176, 0.4218688905239105, -1.3772343397140503, 0.6376263499259949, -0.7495329976081848, -1.2127633094787598, -2.0205559730529785, -0.6264670491218567, 0.8346061706542969, -1.8551863431930542, 0.1995314210653305, 0.12245818972587585, -0.10017743706703186, 0.4718346893787384, 0.4146355092525482, 0.8835687041282654, 1.4121320247650146, -0.5377125144004822, -0.3850082755088806, -0.24017301201820374, -0.37894245982170105, -0.5701131224632263, 1.337538242340088, 0.05028665438294411, 1.5792319774627686, -0.6772406697273254, 1.4945406913757324, -2.4898765087127686, -0.09894108772277832, -1.2850372791290283, 1.03602933883667, 0.1383531242609024, 0.2226540446281433, 0.8069690465927124, -0.554062008857727, 1.0764750242233276, -0.5342011451721191, 0.12521947920322418, -1.744032859802246, -0.09178679436445236, 0.5294960141181946, -0.518106997013092, 0.05742758512496948, 0.4512872099876404, -0.6179493069648743, -0.28680625557899475, 0.48969849944114685, 0.5645240545272827, -0.7101075053215027, 0.03611040115356445, 0.7817474603652954, -1.027714729309082, 0.38165634870529175, 0.6692826151847839, -0.19940680265426636, 1.4674626588821411, 0.546524167060852, -1.1730897426605225, 0.4622606337070465, -0.640198826789856, 0.5919771194458008, 0.963767409324646, 0.6168634295463562, -1.8859370946884155, 0.5010582208633423, 0.9118617177009583, -0.955123782157898, -0.16399984061717987, -1.1657094955444336, 0.996637225151062, -0.03885940834879875, 0.6569345593452454, -1.7872636318206787, 0.2606881260871887, 0.4332294762134552, -0.33823221921920776, 1.3405576944351196, 0.3236987888813019, -1.647968053817749, 1.1177908182144165, -0.6198771595954895, -1.5017436742782593, -0.06105878949165344, -0.4794391691684723, -0.7661088705062866, -0.7467109560966492, -0.5688545107841492, -0.23265282809734344, 1.934253215789795, 0.024724772199988365, 0.6017385721206665, -0.4680939316749573, 1.4833773374557495, 0.756000816822052, 0.6220926642417908, 0.11544868350028992, -1.8186888694763184, 0.8265681862831116, -1.0186479091644287, 0.5980070233345032, -0.07067861407995224, -0.6962642073631287, -0.48328885436058044, 1.7190877199172974, -0.9448615908622742, 0.42736339569091797, 0.2981100380420685, 0.4341236650943756, -0.8930388689041138, -0.3944256603717804, 0.7889336347579956, 0.07108650356531143, -0.8674630522727966, -0.09962628781795502, -1.0081524848937988, -0.6460353136062622, -0.8132342100143433, -0.308807909488678, 0.4140917956829071, -0.5905994176864624, -0.8515940308570862, -1.1054799556732178, 1.5682798624038696, 0.5292061567306519, -0.5167486667633057, 0.7010810971260071, 0.4382133185863495, 0.27147993445396423, 0.013601591810584068, -0.3501683175563812, -2.3833999633789062, 0.5804242491722107, -1.2700519561767578, -0.36355555057525635, 0.9223794341087341, 1.741575837135315, 0.3180428445339203, -1.4614012241363525, 2.890799045562744, -1.0145858526229858, -0.08856818079948425, -1.8374234437942505, 0.3204047977924347, 0.006467742845416069, -0.5112225413322449, 1.2763652801513672, -0.285663902759552, 2.2498152256011963, 0.20300279557704926, -1.3549517393112183, 2.903710126876831, -0.389947772026062, 1.3772578239440918, -0.7074908018112183, -0.2515431046485901, 0.0006623665685765445, 0.11969683319330215, 0.7548921704292297, 0.1755162626504898, 0.026497408747673035, -0.8645745515823364, -0.7792800068855286, -0.5021249651908875, 1.5654460191726685, -0.1630108654499054, 0.9123624563217163, 0.48702630400657654, 1.9480196237564087, -2.5468947887420654, -1.2284173965454102, -0.22684995830059052, -0.4172235429286957, -1.7277917861938477, 0.6123204827308655, 0.7050514817237854, 0.0651370957493782, 0.8419767618179321, -0.5792302489280701, -0.43968576192855835, -0.22476962208747864, 2.5964133739471436, -0.8548159599304199, 0.9306305050849915, 1.0177313089370728, 0.9566784501075745, -0.010472374968230724, 0.8046355843544006, 0.5256679654121399, -0.3974497616291046, 0.9085131287574768, 0.40173229575157166, 1.2173004150390625, 0.18879050016403198, -0.3757455348968506, 0.6780380010604858, -1.561173915863037, 0.45273828506469727, -0.12274889647960663, -0.18817386031150818, -0.5033085942268372, -0.7194358110427856, -0.21592074632644653, 0.19592231512069702, -2.2931995391845703, -2.2533228397369385, -0.17989294230937958, 0.8234382271766663, 0.5677955150604248, -0.11776965111494064, -1.3064908981323242, 0.07008028030395508, -1.2934728860855103, 0.812481701374054, 0.296724796295166, -0.1066059097647667, -1.5356799364089966, 0.986655592918396, -0.18592022359371185, -0.7720301747322083, 0.530761182308197, -0.4746948480606079, 0.6719748377799988, 1.423980712890625, 0.6873325705528259, -0.06493635475635529, 0.9442987442016602, -1.5178964138031006, 1.2435981035232544, -0.10026851296424866, -1.002963662147522, 0.3195815086364746, -0.7499675154685974, -0.33129367232322693, -2.0676705837249756, 0.2469039410352707, 0.5637841820716858, -0.8070122599601746, -0.7043309807777405, -0.05144370347261429, 0.6627563238143921, -0.3616584241390228, 0.20278842747211456, 0.9322398900985718, -1.508184552192688, 0.056063391268253326, 1.664916753768921, -0.6129484176635742, -1.411533236503601, 1.4086579084396362, 1.7644773721694946, -0.49033185839653015, -0.5802837610244751, 1.5628948211669922, 0.10736536234617233, 0.8302851915359497, -0.3476814329624176, -0.10439439862966537, -0.31734928488731384, -0.25870370864868164, -0.9518654942512512, -0.09497839957475662, -0.09880579262971878, 0.07245905697345734, 0.15410463511943817, -0.7257251739501953, -0.23125295341014862, 1.9056572914123535, 0.06120920926332474, 0.7989774346351624, 1.574413776397705, 0.24356518685817719, 1.2714289426803589, -2.25085711479187, -0.4200921654701233, 1.025931715965271, -1.1905343532562256, -0.481654554605484, -0.710997998714447, 1.3873151540756226, 0.9713565111160278, 0.1295081228017807, -1.487607717514038, -2.0982017517089844, 0.6176844239234924, 0.11429708451032639, 0.04868670180439949, 0.8056240081787109, -0.5766595602035522, -1.9343667030334473, -0.22477278113365173, -0.09068218618631363, -0.37413227558135986, 0.5055570006370544, 0.9722191691398621, 1.0488744974136353, 1.1580584049224854, 0.20435640215873718, -0.14506347477436066, 1.1947898864746094, 1.606769323348999, -0.33454054594039917, -0.4640224874019623, 0.3519252836704254, -1.3468204736709595, 2.13440203666687, -0.24077573418617249, 0.8104085922241211, -0.38178497552871704, -0.4660049378871918, 0.021096672862768173, -0.054457277059555054, 0.035228557884693146, 1.4303061962127686, 0.14603333175182343, -1.0169627666473389, 0.40363606810569763, 1.0401190519332886, 0.6273111701011658, -0.6991178393363953, 0.6329575181007385, -1.035144329071045, 0.8959535360336304, 1.186657428741455, -0.629920244216919, -0.21809923648834229, 0.07490634173154831, 0.09050726890563965, -1.2786149978637695, -0.9833446741104126, 0.699069619178772, 0.9979784488677979, 0.2567056715488434, 0.2991240322589874, 0.9373394250869751, -1.4072774648666382, 0.7694955468177795, 0.5055976510047913, 1.180544137954712, 0.2580602169036865, -0.6837441325187683, 0.28208673000335693, -0.7940820455551147, 0.3147694766521454, -0.8038347959518433, 2.0735905170440674, -1.2827277183532715, -2.9463510513305664, -0.5893749594688416, -0.37470588088035583, -0.3753049671649933, -0.4806970953941345, -2.399838447570801, 1.187665343284607, -1.9899675846099854, -0.671781599521637, -0.32565006613731384, -0.8412421345710754, -0.1866946816444397, -1.1019704341888428, 0.9309089779853821, 0.8454996347427368, 1.7639188766479492, -0.3970418870449066, -0.38661906123161316, 0.6875209212303162, 0.3614453673362732, -0.2654542624950409, 0.3288795053958893, 1.4766016006469727, -0.5490632653236389, -0.6530594229698181, -1.9565117359161377, -2.3659486770629883, 0.4911513924598694, 0.4467177391052246, -1.0544265508651733, 0.6108120679855347, -1.3300825357437134, -2.7082924842834473, -0.6075897812843323, 1.8476848602294922, 0.27629780769348145, 1.2790979146957397, 0.406552255153656, -1.6418293714523315, -1.0993239879608154, -1.2461779117584229, 0.4514330327510834, 1.4667284488677979, 0.23874104022979736, -0.39436277747154236, -0.21431738138198853, 0.2687450349330902, 0.38466835021972656, -0.30053168535232544, 0.783581018447876, 0.3448430299758911, -0.1096547544002533, -0.08272416144609451, 1.1291159391403198, 1.186430811882019, 3.472658634185791, -0.2376752495765686, 0.5560086369514465, -0.37760376930236816, -0.9052854180335999, 0.7355467081069946, 1.1196843385696411, 1.55942702293396, -1.3088405132293701, 0.574350118637085, -1.7418993711471558, 0.1819346398115158, 0.37035542726516724, -0.18449921905994415, 0.9002995491027832, -0.3525978922843933, -1.1223067045211792, -0.08066429942846298, 1.3334476947784424, -0.06880278140306473, -1.3734186887741089, 1.1135789155960083, 0.5662068128585815, -1.1494505405426025, 1.3617738485336304, 0.25950443744659424, 1.5001404285430908, 0.41006630659103394, 0.06967949122190475, 0.10160858929157257, 1.1404114961624146, 0.5278995037078857, -0.4439241886138916, -0.13848386704921722, 0.7074174284934998, -0.19164784252643585, 0.7113953828811646, 1.259846568107605, 0.4594864845275879, 0.4056175947189331, 0.22139260172843933, 0.1761060506105423, -0.8254379034042358, -0.6211549639701843, 1.5680955648422241, -0.15887127816677094, -0.5083407163619995, -0.7406096458435059, -0.5545029044151306, 0.4247971773147583, -0.17500747740268707, 0.16477441787719727, 0.3409941494464874, 0.6723515391349792, 0.6390177011489868, -0.1061917096376419, -0.756343424320221, -0.6031806468963623, -0.4046054780483246, -0.29104679822921753, -1.2758649587631226, 0.6634371280670166, 0.5147149562835693, 1.5453568696975708, 0.5300949215888977, 1.3580158948898315, 0.7455469369888306, 0.4301551878452301, 1.1012096405029297, 0.015242166817188263, -0.6353300213813782, -0.5399318337440491, 0.21672476828098297, -0.10108432173728943, -1.5118260383605957, 2.1894004344940186, -0.49859121441841125, 0.9927247762680054, -0.000709689746145159, 1.2637978792190552, -2.8152711391448975, 1.0405678749084473, -1.0503499507904053, -2.108211040496826, 1.2263202667236328, -0.24160327017307281, 0.31338030099868774, 0.5349027514457703, -0.8853378891944885, 1.013066291809082, -3.1209676265716553, 0.771799623966217, -0.6655232906341553, 0.12256116420030594, 0.5775198936462402, 0.46096092462539673, 0.13193929195404053, 1.7868657112121582, -0.6899571418762207, 0.11612356454133987, -1.2527987957000732, 0.28822654485702515, -0.937985360622406, -0.6129767298698425, -0.49013569951057434, 1.0300495624542236, -0.2602049708366394, -1.2828166484832764, 0.949435830116272, 0.4196622669696808, -1.2451725006103516, 0.8152793645858765, 0.35540470480918884, 0.580829918384552, -0.13944610953330994, -0.6776841282844543, -1.1050924062728882, -2.0570948123931885, -1.8622386455535889, 0.33482852578163147, -1.9534116983413696, -1.0295264720916748, 0.5415921211242676, 0.5432489514350891, 1.4241337776184082, -0.5462995767593384, -0.9128013253211975, -0.38245123624801636, -0.06249772012233734, -2.1649930477142334, -1.0477294921875, -1.265151023864746, 1.0942264795303345, 0.6234534978866577, 0.9354414343833923, 1.737524151802063, 1.7742475271224976, 0.5591439008712769, 0.10648895055055618, -0.04576756805181503, 0.4393882155418396, -0.43977442383766174, -0.9572300910949707, -1.7821160554885864, 0.08326888084411621, 0.12539871037006378, -1.0151702165603638, -2.016627073287964, 0.6732808947563171, -0.2594904899597168, -0.29858362674713135, -1.1976929903030396, 0.3828844726085663, 0.09723736345767975, 1.6281969547271729, 0.5362059473991394, 2.8119850158691406, 1.3110982179641724, -0.28113046288490295, 0.5424460768699646, 0.6220757961273193, -1.3235456943511963, -0.023589782416820526, 0.33552733063697815, -1.102694034576416, -0.09327641874551773, 0.6289873719215393, -0.3562372028827667, 0.7984800338745117, 0.8284211158752441, -1.609226107597351, -0.7181670665740967, 0.529486894607544, -2.5245134830474854, -1.2842974662780762, -1.0544646978378296, -1.2449415922164917, -0.35430100560188293, -0.9329961538314819, 0.6404509544372559, 1.4733433723449707, -1.3722978830337524, -2.1244595050811768, 0.43183019757270813, -0.4324404001235962, -2.0382437705993652, 0.13718196749687195, 0.8270050287246704, -1.2434715032577515, -2.5383169651031494, 1.0760351419448853, 0.638940691947937, -0.9670530557632446, -0.2745857834815979, 1.573513388633728, 0.495488703250885, 0.5141090154647827, 0.5941974520683289, 0.20361879467964172, -1.1355546712875366, -0.2688499689102173, 1.3281571865081787, -1.6452234983444214, 0.1860736459493637, 1.2553720474243164, -0.3783058226108551, 0.7332251667976379, -1.3274117708206177, 0.9981487989425659, 1.544338345527649, 1.2224587202072144, 0.2440367043018341, -0.9384388327598572, -0.5216350555419922, -0.8384665846824646, -1.2825452089309692, 0.27463358640670776, -1.2744442224502563, -0.1069968193769455, -0.937978208065033, -1.1597965955734253, 1.7130167484283447, -0.6873373985290527, 0.3300715386867523, 0.875302255153656, -0.47564050555229187, -1.428608775138855, -1.3259189128875732, -0.5671193599700928, 0.7486429214477539, 0.5501924157142639, 0.4090327024459839, 0.6679419279098511, -1.7784247398376465, -0.8993415236473083, 1.5962620973587036, -0.4674179255962372, 0.7023543119430542, -0.7772472500801086, -0.6111037731170654, -0.22303226590156555, -1.0957555770874023, -0.261436402797699, 1.109619140625, 0.4780186414718628, -1.2933834791183472, -0.9230931401252747, 0.06191347911953926, 0.3388771414756775, -0.2641655206680298, -0.20607705414295197, 0.06221364438533783, -1.5961517095565796, -0.22314247488975525, 0.41911178827285767, 2.000272035598755, 1.4536992311477661, 0.20319247245788574, -0.040596965700387955, 0.6508929133415222, -0.7623186707496643, -0.5066664814949036, 0.12004660815000534, 0.5236789584159851, 0.47063741087913513, 1.2893062829971313, -1.5760611295700073, -0.14139065146446228, -0.48825782537460327, 0.044828616082668304, 0.4654078781604767, 0.03709684684872627, -0.9260890483856201, 1.3821601867675781, -1.8640249967575073, -0.11224440485239029, 1.96340012550354, 1.3986258506774902, -1.0326440334320068, 0.6805965900421143, 0.9563294649124146, -0.07096671313047409, -1.0306302309036255, 0.3637060523033142, 0.5989248752593994, 0.8517127633094788, 0.8493737578392029, 1.075500726699829, -2.8861944675445557, -0.0941905677318573, 0.14401620626449585, 0.8504572510719299, -0.10903232544660568, -0.32512423396110535, 0.16039936244487762, -0.8093538880348206, -0.30716070532798767, 0.7805773019790649, 0.05048613250255585, 1.008212924003601, 1.2708104848861694, -0.6720260381698608, -0.977713942527771, 0.5349984765052795, -1.4721728563308716, -0.00470732431858778, -0.6246061325073242, -1.9889768362045288, 0.8045269250869751, 1.5549485683441162, 0.66209876537323, 1.3468254804611206, 0.16554373502731323, 0.9092332124710083, 0.7109639644622803, -0.2857133150100708, -0.2297257035970688, 0.20015861093997955, 1.1619526147842407, 0.33579012751579285, 0.31008827686309814, 0.6771731376647949, 0.5733920335769653, -0.4217274487018585, -2.1252715587615967, 1.1766568422317505, -0.9932176470756531, -0.16909794509410858, -0.10713900625705719, -0.9142806529998779, -0.19927053153514862, 0.5616389513015747, -0.8835487365722656, -0.14025066792964935, -2.516456365585327, -1.7940682172775269, 2.1533119678497314, -1.340981364250183, -1.063286542892456, -0.252234548330307, -1.405061960220337, -1.19605553150177, 1.0094164609909058, 0.5057604908943176, 0.7570382952690125, 0.09802445769309998, -0.1763220727443695, 0.39267870783805847, -0.35243892669677734, -0.8146888017654419, -0.9453751444816589, 0.06046808511018753, 0.13671137392520905, 0.9543991088867188, -0.42326658964157104, -0.03766033798456192, 0.8263537883758545, 1.9809503555297852, 0.8560349345207214, -0.3241046667098999, -0.5472318530082703, 0.12521901726722717, -1.6963653564453125, 0.18426313996315002, 1.4325300455093384, 1.5204120874404907, 1.553784966468811, -0.6140615344047546, -0.11498961597681046, -0.5478566884994507, 1.30730402469635, -0.873009204864502, 0.12108681350946426, 1.4232380390167236, -1.4247459173202515, -0.5376112461090088, -0.1606420874595642, -0.6942034363746643, -0.026852479204535484, -0.18621109426021576, 0.29139259457588196, 1.0519461631774902, -2.3880176544189453, -0.7732218503952026, 0.8399017453193665, -0.48247238993644714, 1.4721941947937012, 1.0177549123764038, -1.0514423847198486, -0.5236603021621704, 0.5041117668151855, 0.643478512763977, -0.6101315021514893, 0.008680715225636959, 1.286653995513916, 0.8650944232940674, 0.6333903074264526, 0.5959562063217163, -0.08713871240615845, -0.5907439589500427, -0.9199921488761902, 0.7914505004882812, 0.684354305267334, 0.4455743730068207, 0.7129718065261841, 0.3738902807235718, 0.7073391675949097, -0.9555045366287231, -2.2718570232391357, 1.4332287311553955, -2.0954477787017822, 1.921453595161438, -0.5617090463638306, 0.5642607808113098, 0.5956630706787109, -1.235292911529541, 0.05475037172436714, 0.1909332424402237, -0.2762542963027954, 0.33881229162216187, -0.9658892750740051, 0.8544120192527771, -0.0628274530172348, -0.09542061388492584, 0.06411562114953995, -0.27543967962265015, -2.1599981784820557, 0.27013421058654785, -1.43477463722229, -0.9511139988899231, 0.1466921865940094, 0.46696001291275024, 0.979926586151123, -0.6001498699188232, -1.5946550369262695, -1.6493686437606812, 1.5741363763809204, 3.031984567642212, 1.1966180801391602, 0.10249882936477661, 0.2526291012763977, 0.0186762735247612, 1.1195224523544312, -1.2555278539657593, 1.524450659751892, -0.7171485424041748, 0.42724743485450745, -1.4582507610321045, 0.686223566532135, -0.5257527232170105, -0.3230735957622528, 0.4688703119754791, -0.4228736460208893, 0.39708736538887024, 0.5993943214416504, -1.2937978506088257, 2.452860116958618, 0.8509413599967957, -0.3898411989212036, 0.05525472015142441, -1.288220763206482, -0.3602960407733917, -0.39281338453292847, 0.4363144338130951, 1.6116564273834229, -3.0960772037506104, -1.2345119714736938, 0.8663741946220398, -0.6682270765304565, 0.11099138855934143, -0.6415477395057678, -0.5139633417129517, 0.13472965359687805, -0.6882264018058777, 0.945315957069397, 0.0838390365242958, -0.5928707718849182, 0.5659360289573669, 0.3496900498867035, 0.4403100609779358, 0.07172343134880066, -0.7373051047325134, 0.735497772693634, 0.3531217873096466, 1.5278217792510986, 0.45803266763687134, 0.09158655256032944, -0.11428719013929367, 1.6811529397964478, 1.245157241821289, 0.47418227791786194, 0.15172208845615387, -0.20427162945270538, 1.260411024093628, 0.07329604774713516, 0.14856493473052979, 1.1925231218338013, -0.8271152377128601, -0.3306506872177124, 0.871853232383728, -1.1940181255340576, 1.2579907178878784, 0.019549770280718803, -1.0042064189910889, -0.6114026308059692, 1.1183096170425415, -0.2957652509212494, 0.7276806235313416, -0.030951816588640213, 0.09911801666021347, -1.8026217222213745, 0.2105068415403366, -0.20658054947853088, -1.3216068744659424, 0.5924302935600281, 1.2172913551330566, -0.4474961459636688, 1.9711214303970337, 1.0579209327697754, 1.094194769859314, 0.9815509915351868], [-0.6044626832008362, -1.6377497911453247, 0.9689397215843201, -0.5614743232727051, 1.310788869857788, 1.3616517782211304, 0.8319703936576843, 0.8440966010093689, 0.056920792907476425, 0.5814581513404846, -1.368384599685669, 0.26699337363243103, 1.6444379091262817, 1.3987618684768677, 0.5389714241027832, -0.2529236078262329, 0.263137549161911, 0.7617784738540649, -0.009395426139235497, -0.9517567753791809, 1.1344460248947144, -0.3413453996181488, -1.1230602264404297, 0.7294318079948425, 0.16961954534053802, 1.205241322517395, -0.39088281989097595, -1.6957730054855347, -1.0678503513336182, -0.5014811754226685, 0.6863410472869873, 0.37471896409988403, -0.42371147871017456, 0.6642744541168213, 1.3640583753585815, -0.7363938093185425, -1.2056063413619995, 1.3388628959655762, -0.0721975564956665, -1.2745338678359985, 1.3493977785110474, 0.0922999456524849, -0.07154364138841629, -1.0119268894195557, 0.983695387840271, 0.11860277503728867, -1.1985490322113037, -0.9128647446632385, 2.2067198753356934, -0.10678404569625854, 0.14331410825252533, 0.3818962275981903, 2.233755588531494, 1.694806694984436, 0.5050402879714966, 2.265345573425293, -1.6264660358428955, -1.381807565689087, 1.8844702243804932, -0.5337783098220825, 1.9185901880264282, 0.07632213085889816, 0.8401864171028137, -0.5041608810424805, 1.1543902158737183, -0.0018487380584701896, -0.6920021772384644, 0.2091650813817978, 1.7095329761505127, -1.0174970626831055, 1.110663652420044, -0.7248170971870422, 0.28756627440452576, -0.678417980670929, 0.8099597096443176, -1.7978349924087524, 0.5050617456436157, -0.05020018294453621, -0.03139939531683922, 0.3216071128845215, 1.2485181093215942, 0.42958974838256836, 1.228232979774475, -1.196358323097229, -0.8476895689964294, -0.8240605592727661, -1.9893686771392822, -2.6338107585906982, -0.40542683005332947, -0.2038774937391281, -1.5576632022857666, 2.707746744155884, 0.4953979253768921, -0.2831646800041199, -1.5375477075576782, 0.7625539898872375, -0.3730829954147339, 0.6830399632453918, 0.33146920800209045, -2.636828899383545, -1.5646814107894897, 1.8070615530014038, -0.6576986908912659, 0.2349223643541336, -0.2175518125295639, 0.4995623826980591, 0.14335329830646515, 1.1273072957992554, -0.3170643150806427, -0.4483668804168701, 0.4977291524410248, -1.6135073900222778, -0.5328216552734375, 0.5742985010147095, 0.29518675804138184, 1.8276101350784302, -0.918034553527832, -0.2184564620256424, 0.3709798753261566, 0.7910935878753662, -0.879802942276001, 0.4704333245754242, -0.7616347670555115, 0.17550309002399445, 0.6385511159896851, -1.4649710655212402, -1.8663588762283325, -2.009108304977417, 1.521621584892273, 1.1088436841964722, 0.8685813546180725, -0.43128877878189087, -0.0025624625850468874, 1.1813600063323975, 2.8678483963012695, 0.3725336492061615, -1.0713785886764526, 0.276140034198761, -2.0763049125671387, 0.9161584973335266, -2.354288101196289, -0.1852370947599411, 1.1214029788970947, -0.3255190849304199, -1.6139190196990967, 1.138207197189331, 0.7814207673072815, -0.23549200594425201, 0.3168145418167114, 0.6682233810424805, 0.36504659056663513, 0.3426770865917206, -0.4855581223964691, -0.9775448441505432, 0.3813003897666931, 0.2973001003265381, 0.5824864506721497, -2.671722412109375, -0.6341135501861572, -1.747514009475708, 1.3354929685592651, 0.4901079833507538, -1.4638283252716064, -1.4757965803146362, 1.148790717124939, 0.9086839556694031, -0.12994889914989471, -1.4657769203186035, -1.65243661403656, -0.11541497707366943, 1.1293056011199951, -1.0031331777572632, 0.03732869774103165, -0.7366682887077332, 1.0978612899780273, -0.0512530617415905, 0.29238802194595337, 0.936994731426239, -1.1987724304199219, 1.1953569650650024, 0.07238695025444031, 2.0462348461151123, -0.8273081183433533, -1.4044229984283447, -0.11786385625600815, -0.8992334008216858, 0.09542736411094666, -0.6357756853103638, 1.8622370958328247, -0.7263772487640381, -0.9169318079948425, 0.9523611664772034, -0.5035061836242676, -0.7046507000923157, -0.5354177355766296, -1.598583698272705, 0.11188866943120956, -1.3461259603500366, 0.4793722629547119, -1.5259695053100586, -1.4540941715240479, -0.4386311173439026, -1.6290326118469238, 0.06715290993452072, 1.039617896080017, 0.11627786606550217, -0.09967292100191116, 1.8341072797775269, 0.4138229489326477, 1.5383086204528809, 0.16648481786251068, -1.1150420904159546, -1.7593371868133545, 0.9091728925704956, 0.7374382615089417, 0.3201826810836792, 0.21312682330608368, -0.10906338691711426, 1.52854585647583, 0.024253161624073982, -1.2334809303283691, -0.5674012303352356, 0.3539819121360779, -0.5270602703094482, -0.843684196472168, 0.6242575645446777, 1.2202153205871582, -1.0918974876403809, -0.4703613817691803, -1.0917561054229736, -1.0651530027389526, 1.7205727100372314, 0.2792213261127472, 0.6293507814407349, 0.45090991258621216, -0.5876275300979614, 0.7225354313850403, 0.40990403294563293, -0.49079084396362305, -0.7551533579826355, -0.45534980297088623, 0.6557270288467407, -1.596950888633728, -0.37373146414756775, 0.9378528594970703, 0.4101468324661255, -0.3486638069152832, -0.41553783416748047, -0.523430347442627, 0.7712150812149048, -0.9681560397148132, 0.4609230160713196, -1.1450986862182617, -0.2295840084552765, -0.13194173574447632, -1.9822667837142944, 0.44458821415901184, 0.6117966175079346, 0.328512966632843, -1.067467451095581, -1.655064582824707, 0.9867684245109558, -1.0589126348495483, -0.2739175856113434, 0.5045469999313354, 1.7247780561447144, -0.5149369835853577, 0.19288745522499084, 0.21356187760829926, 0.18391209840774536, 0.07225310802459717, 1.7710479497909546, 0.4103752374649048, 0.034945517778396606, -0.23474828898906708, 0.4827706217765808, 0.959852933883667, 0.04353732988238335, -0.4419541358947754, 0.9671969413757324, -0.7063921689987183, -0.05009535700082779, 0.5327703952789307, 0.188252791762352, -0.08366603404283524, -1.2576013803482056, -0.2257469892501831, -1.3547883033752441, -1.3272769451141357, 0.1791437417268753, 1.5544952154159546, -0.3017066717147827, -0.3356184661388397, -1.849320411682129, -1.292584776878357, -0.6843328475952148, -1.1851415634155273, 0.13271218538284302, 0.8576561808586121, 0.17101652920246124, 0.5325714945793152, -1.6431193351745605, -1.0736069679260254, 1.2444753646850586, -0.0702696442604065, 1.1387721300125122, 0.39525526762008667, 0.6783138513565063, -1.5139784812927246, -0.3308345675468445, 0.39876872301101685, -0.7937433123588562, 1.3891150951385498, -0.22294561564922333, -0.6097841858863831, 0.22956381738185883, -0.9270326495170593, 0.35129210352897644, -1.5687005519866943, 1.3009655475616455, 0.7280656099319458, -0.4025305509567261, -0.07695679366588593, 0.13925917446613312, 0.883454442024231, 0.15674397349357605, 0.00840833131223917, 0.7371047735214233, 0.46781978011131287, 0.5101587176322937, -0.49623730778694153, 0.36164000630378723, 0.2234615981578827, 2.4640305042266846, -1.4635311365127563, 0.8485923409461975, -0.2235913872718811, -1.7030844688415527, -0.9237419962882996, 0.39705219864845276, 0.10391257703304291, -2.5999813079833984, 0.5801586508750916, -0.1634954959154129, -0.06185600906610489, 1.531076431274414, 0.7986128926277161, -0.16000479459762573, 0.6713221073150635, -1.0974321365356445, 0.7529186606407166, 0.9928314685821533, -0.19999927282333374, 0.7627480626106262, -1.4162880182266235, -0.9668310880661011, -0.6344666481018066, -1.0751962661743164, -1.414463758468628, 0.1415392905473709, 0.07969678193330765, -1.5230191946029663, 0.8315925598144531, -0.06297343224287033, 0.25287944078445435, -0.4589815139770508, 1.4870291948318481, -0.5852645039558411, 0.9337384700775146, -0.17179431021213531, 0.6702761054039001, 0.6171467900276184, 0.5317786931991577, -0.2248896360397339, -0.855963408946991, -0.8675466775894165, -0.6776934266090393, -0.08181419968605042, -0.1662953644990921, -1.7067173719406128, -0.9462242722511292, -0.7458844780921936, 0.12099438160657883, 0.16675600409507751, -1.1017905473709106, 0.03851990029215813, 0.8305248022079468, 2.0154478549957275, -0.3289792537689209, 0.3149668276309967, -0.08031736314296722, 1.0193246603012085, 1.321489930152893, -1.648271083831787, -0.5450271964073181, 0.33106762170791626, 0.3353251516819, -0.1395290046930313, -1.8372466564178467, -0.03723737597465515, 1.7662626504898071, 0.2823959290981293, 0.4312517046928406, -0.4300950765609741, -0.9448840022087097, -0.16550005972385406, -0.12565448880195618, -0.8007165193557739, 1.419782280921936, -1.5829914808273315, -0.5381518006324768, 0.9389327764511108, -0.1641935408115387, -0.5488606691360474, -0.7959967255592346, 0.6480394005775452, -0.18158718943595886, 0.42222467064857483, -0.4276658296585083, 0.2775730788707733, 1.3786938190460205, 1.0153815746307373, 0.8024393320083618, 0.8501102328300476, 0.7041082382202148, -0.8017735481262207, 0.6727638840675354, -0.7952565550804138, -0.1915971338748932, -0.059246983379125595, 0.26790282130241394, 1.527398705482483, -1.2747806310653687, 0.03758927807211876, -0.5124343037605286, -1.4755549430847168, 0.883368730545044, -1.0817557573318481, -0.8830487132072449, -1.6189466714859009, 0.139499694108963, -1.8095191717147827, 1.005332112312317, -0.8272389769554138, 0.21907153725624084, 0.6961020827293396, -0.43468597531318665, -0.10920806229114532, 0.8671651482582092, 0.8035192489624023, -0.0007729968056082726, 1.8083103895187378, 1.2683731317520142, -0.6842567920684814, 0.9345962405204773, -0.21125896275043488, -1.832218885421753, 0.757617175579071, 2.6577794551849365, -0.5793530344963074, 0.449410080909729, 1.8887399435043335, -0.7099089026451111, 0.741477906703949, -2.197188377380371, -1.7352572679519653, 0.3738115429878235, 0.36102065443992615, 0.6546276807785034, -0.7586699724197388, 0.005804519169032574, 0.19388657808303833, -1.0231517553329468, -0.7432827949523926, 0.9951436519622803, -0.29950591921806335, 0.43142396211624146, -0.9726665616035461, -0.46949222683906555, -1.269788146018982, -1.5292223691940308, -1.8935530185699463, -0.28574326634407043, -1.203271746635437, -0.04879892244935036, -0.04595364257693291, -0.07114575058221817, 0.2546892464160919, 0.691413164138794, -0.7669751048088074, 0.4254927337169647, 0.6586612462997437, -1.1472634077072144, -0.09845225512981415, -1.6002960205078125, 0.12309014052152634, 1.8082735538482666, 1.4117941856384277, -1.0690675973892212, -0.48765814304351807, -0.3371226191520691, 0.30915772914886475, 1.7983359098434448, 0.7649310827255249, 0.29148951172828674, -0.02441626414656639, 0.4361344575881958, -0.13637375831604004, -0.0643112063407898, -1.33900785446167, -1.8998008966445923, -1.2738661766052246, -0.7604042291641235, 1.2775602340698242, 1.443129539489746, -0.31747275590896606, -0.07896384596824646, -0.3320917785167694, 0.1681421846151352, 1.3983043432235718, -0.9112439751625061, 0.5389727354049683, 2.3651607036590576, 0.32273921370506287, -0.4629588723182678, 0.28607651591300964, -0.3642564117908478, 1.4218635559082031, 1.2889881134033203, 0.4682307243347168, -1.3493763208389282, 0.26852554082870483, 1.0881294012069702, 0.1451517790555954, -0.6892509460449219, 0.38902878761291504, 0.2525383234024048, 0.2826921045780182, 1.0130479335784912, 0.4056242108345032, 0.8235544562339783, -1.1944259405136108, 1.7310553789138794, -0.7484086751937866, 1.6284263134002686, 1.0601447820663452, -2.4825644493103027, -0.6441032290458679, 0.11955195665359497, 0.26101791858673096, 0.5171682238578796, 0.28809383511543274, 0.18168720602989197, -0.2740972638130188, -1.0595312118530273, 0.2879302203655243, -0.07378071546554565, -0.8697444200515747, -0.47012269496917725, 1.7922309637069702, -0.04495808482170105, 0.9707984924316406, 0.662471354007721, 0.45905178785324097, -0.9265314936637878, -3.3229987621307373, -0.9434036016464233, 0.4247889816761017, 0.33568087220191956, -0.996756374835968, -1.4696860313415527, -0.5955381989479065, 1.5639986991882324, -0.6250077486038208, 0.05267878249287605, -0.6661828756332397, 0.35345810651779175, 1.5270106792449951, -0.16082267463207245, 0.575502336025238, 1.2821801900863647, -1.7948745489120483, -1.1594935655593872, -0.5285476446151733, 1.3708194494247437, -1.464377522468567, 1.2236201763153076, -0.7376181483268738, 0.7470082640647888, -0.03443129360675812, 0.10390441864728928, 0.7830506563186646, -0.6305486559867859, -0.35207560658454895, -0.08909729868173599, 0.3077719211578369, 1.887962818145752, 0.2664867341518402, -1.0955201387405396, 1.0706321001052856, -0.566447913646698, 0.5841774344444275, 0.4913022220134735, -1.873867154121399, -0.07428456097841263, -0.9711838364601135, -1.8702818155288696, -0.16196079552173615, -0.6633912920951843, 0.3120686113834381, -1.2651655673980713, 0.663528323173523, 0.0346447117626667, -0.10736910253763199, -0.37679994106292725, -1.2379761934280396, -1.3389043807983398, 0.13976901769638062, -0.0018134706187993288, 0.4637136161327362, -0.6446275115013123, 0.6874933838844299, -0.625423789024353, -0.5572355389595032, -0.4883580505847931, 1.365484595298767, 1.181864857673645, 1.449031949043274, 0.7566026449203491, -0.2758995592594147, -0.5551122426986694, 0.32302361726760864, -2.2466092109680176, 0.11967121809720993, -1.5863991975784302, 1.8094850778579712, -2.3375580310821533, 0.08901618421077728, -0.35518208146095276, -1.1228129863739014, -1.7562991380691528, -1.4654133319854736, -0.12850917875766754, -0.7463283538818359, 0.9734604954719543, -1.3769152164459229, -0.529968798160553, -0.01648900657892227, 0.30980032682418823, -0.3746126592159271, -2.393929958343506, -1.273716688156128, 0.94541335105896, 0.11656254529953003, -0.08343151211738586, -0.8365118503570557, -1.765677809715271, -1.6667908430099487, 1.2546838521957397, 2.192819118499756, -0.714879035949707, 0.02346714586019516, -0.127478688955307, 1.090030312538147, 0.5182656645774841, -0.09940221160650253, 0.28961384296417236, -1.4319692850112915, -0.22474509477615356, -0.8948500752449036, 0.25308069586753845, -1.2563071250915527, -0.005752136930823326, 0.8287057280540466, 0.6115706562995911, 0.8226718902587891, -0.9099803566932678, -0.6091312170028687, 0.7665690183639526, -0.7912997603416443, 0.6919789910316467, 1.3882900476455688, -0.19193528592586517, -0.5179300308227539, 0.608852207660675, -1.3570992946624756, 0.5534641146659851, 0.41069599986076355, -0.33502325415611267, 0.9627233147621155, 0.7169856429100037, 0.8681260943412781, 0.22931475937366486, -0.2630420923233032, 1.5291287899017334, 1.1508715152740479, 0.7149245142936707, -0.5006228089332581, 0.42445462942123413, 1.5704001188278198, 0.10666728764772415, -1.0711032152175903, 1.7754952907562256, 0.04161767661571503, 0.1956331431865692, -1.0568723678588867, 0.860149085521698, -1.8993587493896484, -0.7523195743560791, -0.0779833123087883, 1.1767836809158325, 0.9595584273338318, 0.8445627689361572, 0.14085198938846588, -1.9930464029312134, -3.349719524383545, 0.8858862519264221, -2.0080249309539795, 0.43217021226882935, -1.3636856079101562, 1.0729663372039795, -0.9390507340431213, -0.34562426805496216, -0.039254557341337204, 0.6870526671409607, 1.2507539987564087, -0.5892371535301208, 0.28996574878692627, -0.5744298696517944, -0.6976718902587891, -1.9392163753509521, -1.3076908588409424, 0.07299380004405975, 1.4715020656585693, -1.0936367511749268, -0.6168129444122314, 0.430719256401062, -0.38398516178131104, 0.12399554997682571, 1.5556336641311646, -2.44781494140625, 1.6395529508590698, -0.1287955641746521, -0.5550787448883057, -1.3412511348724365, 0.1601749062538147, -1.4416146278381348, 0.7413654923439026, -0.7877658009529114, -0.663436233997345, -0.05920005962252617, 1.4405839443206787, -1.3408926725387573, 0.5792003273963928, 1.5596591234207153, -0.07179170846939087, -0.6304728388786316, -0.6453033089637756, -1.1785634756088257, -0.8199756145477295, -0.5468378067016602, -0.7222063541412354, 0.574277937412262, 0.08547580242156982, 0.17973797023296356, 0.20482973754405975, -0.7071120738983154, -1.2998400926589966, -1.4340441226959229, 0.6796353459358215, -0.28945446014404297, 1.822874903678894, -0.9750384092330933, -0.8052234649658203, 0.19951783120632172, 1.5235522985458374, -0.16649433970451355, 0.14571024477481842, -0.3390227258205414, 0.8298024535179138, 0.6603519916534424, -0.07103286683559418, -1.7214232683181763, 0.6493434309959412, 0.34080228209495544, -0.6830335855484009, -0.4173057973384857, -1.6475203037261963, -1.0280895233154297, -0.32282641530036926, -0.19583262503147125, -0.3293441832065582, -0.6872819662094116, -0.6191779375076294, 0.9393466114997864, 0.7686231732368469, -1.5223637819290161, -0.7885189056396484, 0.06428179889917374, -1.439799189567566, 0.9971534609794617, -1.5373142957687378, 0.9176204204559326, 0.7398290038108826, 0.532271683216095, -1.0849436521530151, 0.5063947439193726, -0.6065890192985535, 0.25535497069358826, -0.5060217380523682, 0.11112931370735168, 0.8361594080924988, 0.18892155587673187, 0.35044392943382263, 0.3537290692329407, -1.3740931749343872, -0.899864137172699, -0.9105180501937866, -1.5973823070526123, 1.5065832138061523, -0.7233572602272034, -1.7390930652618408, 0.25504547357559204, -0.18230193853378296, 0.7832626104354858, -0.9341877698898315, -0.35971799492836, 1.691591501235962, 0.7410513162612915, 0.6038898229598999, 2.14326810836792, 1.2706876993179321, -0.22313615679740906, 0.01601909101009369, -1.2966030836105347, 0.09452354907989502, 0.578823983669281, 0.770169734954834, 0.7398000359535217, -1.3472182750701904, 0.6365434527397156, -0.8731123805046082, 0.29333680868148804, 0.29890286922454834, -1.0084267854690552, 0.4639146327972412, -1.3871008157730103, -1.0423121452331543, 0.31452760100364685, -0.8520517349243164, -0.21148715913295746, -0.8544777631759644, -1.4607349634170532, -0.628596842288971, -0.8736138343811035, 0.4389773905277252, 1.1030337810516357, 2.4439704418182373, 0.5287894010543823, -0.5305006504058838, 0.09042902290821075, -0.8558089733123779, 0.29208287596702576, 1.6668123006820679, 0.9147058725357056, 0.11843407154083252, 0.1905914545059204, -1.0031709671020508, 0.7280496954917908, -2.54654860496521, 1.6019476652145386, -0.9453582167625427, -0.8098326921463013, 0.8060076832771301, -1.1246545314788818, -0.20519429445266724, 0.5904375314712524, 0.4321179986000061, 1.8334442377090454, -1.386155128479004, -1.5269795656204224, -0.21243956685066223, -1.5859675407409668, 0.5970261693000793, -0.9417104125022888, 0.2911900281906128, -0.898815929889679, -0.5303707122802734, 0.38193461298942566, 0.27299150824546814, -1.0047756433486938, 0.05112580209970474, 0.2139342874288559, -0.25509703159332275, 0.29277023673057556, -0.42433807253837585, 0.025099212303757668, -1.3917467594146729, -1.1960161924362183, 1.0760526657104492, -0.6012768149375916, 0.7064457535743713, -0.7665858864784241, -1.267673373222351, 0.6737353205680847, -0.5763488411903381, -0.7113623023033142, 0.7114680409431458, 0.1078701987862587, 0.594389796257019, -0.2993077337741852, -0.4712317883968353, 0.14333783090114594, 0.6361852288246155, 1.1084548234939575, 0.1600608080625534, -0.057230524718761444, 0.9872714281082153, -0.17593172192573547, 0.05533428490161896, 2.3562722206115723, -0.21978279948234558, -0.28971123695373535, 0.3594081401824951, 1.929286241531372, -1.2852146625518799, -0.358132928609848, 1.961090087890625, -2.948990821838379, -0.38084709644317627, 0.09872528165578842, 0.551622211933136, -0.2881523370742798, -0.8736342787742615, 1.7170792818069458, -0.863373875617981, -0.03521910682320595, 1.1003758907318115, -0.23162232339382172, -1.6111900806427002, -0.5949956774711609, 1.0497487783432007, 1.1006988286972046, 0.17175136506557465, 0.5670223236083984, -0.14734947681427002, 1.0521726608276367, -0.5273464918136597, 0.008631167933344841, -2.7416234016418457, -1.1183711290359497, 0.14539767801761627, 1.6853207349777222, 1.386155366897583, 1.4758927822113037, -1.0641732215881348, -0.9045279026031494, 0.29281848669052124, 0.46224573254585266, 1.7775483131408691, -1.5078843832015991, 1.2663962841033936, -1.8975318670272827, -0.4396085739135742, 0.204084113240242, 1.4196932315826416, -1.0086835622787476, 1.1742770671844482, 0.6874544620513916, -0.9138559103012085, -0.7957353591918945, -0.10601890087127686, 0.06999893486499786, 2.3188347816467285, -0.33840030431747437, 2.398015022277832, 1.2123134136199951, -0.8176937699317932, -0.48622927069664, -0.16825929284095764, 1.247645616531372, -1.2343113422393799, -0.3166485130786896, -0.6360530257225037, 1.2442554235458374, 2.1225810050964355, 0.4304816424846649, -0.11856044828891754, -0.818683922290802, -0.782335638999939, 0.37018534541130066, 0.21395492553710938, -2.046761989593506, -0.8283028602600098, -0.22603768110275269, 0.06999944150447845, 0.013671625405550003, -0.3093179762363434, -0.7514241933822632, -0.8382283449172974, 2.058760166168213, 0.5083596110343933, 0.49287790060043335, 0.7083113789558411, -0.382865309715271, 0.10215655714273453, -0.5708755254745483, 0.6765574812889099, 0.3536096513271332, 1.3764265775680542, -1.0256794691085815, -1.2305145263671875, 0.6669823527336121, 0.38896822929382324, 0.8424004316329956, 0.46709391474723816, -0.4502409100532532, -0.17123499512672424, 0.5962744355201721, -0.8873657584190369, -0.4595615863800049, -0.3629160225391388, -0.02861836925148964, 0.07927331328392029], [0.04676162078976631, 1.0602704286575317, 0.5494462847709656, -0.3875332176685333, 0.4722111225128174, -0.5013762712478638, 1.4028195142745972, -0.9051231741905212, -1.3164644241333008, 1.155135989189148, 0.8213170170783997, 0.4228796064853668, 1.2992231845855713, 0.02861720882356167, 1.155006766319275, -0.10303909331560135, 2.1761362552642822, 0.9825453758239746, -1.4213379621505737, -0.8515963554382324, -0.03466181457042694, -0.16808627545833588, 0.19910870492458344, -1.344329595565796, -1.229076862335205, -0.18853510916233063, 0.5435348749160767, -0.9475728869438171, 0.31822800636291504, 0.9679960012435913, 0.3040761649608612, -0.62483149766922, 0.025428928434848785, -1.347928762435913, -0.4193485677242279, -0.14575672149658203, -2.5355007648468018, 2.1228513717651367, 0.9456726908683777, -0.18828457593917847, -1.0147294998168945, -0.36707696318626404, -2.0334296226501465, 0.5266087651252747, 0.3681330978870392, 0.13564340770244598, 0.001610257662832737, -1.0314440727233887, 1.9767751693725586, 0.09072400629520416, -0.9396303296089172, -0.854823648929596, -1.0761747360229492, 0.46888479590415955, 0.6223804950714111, -1.5022836923599243, -0.9158501029014587, -1.3126842975616455, -0.04626122862100601, 0.10287287831306458, 0.5862260460853577, 1.5205564498901367, 0.7613674402236938, -1.1639066934585571, -0.8230186104774475, -0.3268372416496277, 0.7439015507698059, 1.4728466272354126, 1.6098041534423828, -0.3087591230869293, 1.7294540405273438, -1.7588917016983032, -0.6258097887039185, -1.3432397842407227, -0.921680748462677, -0.3050423264503479, -0.31143131852149963, 0.1735757440328598, 0.7624847888946533, -0.21154402196407318, 0.19788415729999542, -0.3675419092178345, 0.6096664667129517, -0.8778047561645508, -1.2608524560928345, -0.3276556730270386, -1.4964828491210938, 0.2631831467151642, -1.0638172626495361, 0.007536969613283873, 1.0658425092697144, -0.4090617001056671, -1.028435230255127, 1.9861165285110474, 0.1444849818944931, -0.8561311364173889, 1.825834035873413, -2.1101059913635254, 0.476612389087677, 0.011061446741223335, -0.20178169012069702, 0.8759741187095642, 0.5892759561538696, 0.5269184112548828, -0.6506227850914001, -0.7994707822799683, -1.0161197185516357, 0.0172975305467844, -0.6425555944442749, -1.8012713193893433, -0.048404160887002945, 0.06771685183048248, -0.17172561585903168, -0.2477988600730896, 0.005794766824692488, -1.907450795173645, -0.9265479445457458, -2.375352382659912, -1.4656072854995728, 0.29614895582199097, 1.8271311521530151, -1.1148936748504639, 1.1932194232940674, -1.2778162956237793, 1.1308337450027466, -0.8196766972541809, -2.138216495513916, -0.7146729826927185, -1.6361265182495117, -0.47552555799484253, 0.0013643106212839484, 0.711673378944397, -1.0322256088256836, -0.5890556573867798, -0.7160930037498474, -1.1373622417449951, 0.29269373416900635, 1.4936968088150024, -0.19887185096740723, 1.5971641540527344, -0.9752132892608643, -0.786245584487915, 0.035406164824962616, -0.7483645677566528, 0.7696350812911987, 0.25878214836120605, -1.1534416675567627, -0.464737206697464, -1.261958360671997, -0.33429020643234253, -0.6197177767753601, -0.7134565114974976, 0.0573742501437664, 0.4889322817325592, -0.8908961415290833, 0.765667736530304, 0.6830877065658569, -1.0716938972473145, -0.5237414240837097, 1.9757407903671265, -0.8271777033805847, -0.06865119189023972, 0.4743309020996094, -0.6498018503189087, 0.11877715587615967, -0.4771599769592285, 0.4801292419433594, -1.015696406364441, 0.7823870182037354, -0.723410964012146, -0.5131548643112183, 1.0423520803451538, -1.3131452798843384, -1.0982733964920044, -0.5686410665512085, -0.7377490401268005, 0.21143920719623566, -0.9792677164077759, 0.2666502594947815, 0.8927125334739685, 1.0110996961593628, 0.6038712859153748, -1.0118577480316162, 0.24562837183475494, 1.5042496919631958, -0.3386707901954651, -0.398364394903183, -1.1951267719268799, -0.06557567417621613, 0.02684570848941803, -0.22466935217380524, -0.8790618181228638, -0.09401440620422363, -0.5419424772262573, -0.6771488785743713, -1.599124789237976, 1.2307145595550537, -0.037075724452733994, 0.5077770948410034, -0.9156109094619751, 1.0337671041488647, -0.30070772767066956, 2.0641391277313232, 0.09738833457231522, 1.6666730642318726, -1.3523426055908203, 1.377130150794983, 0.8146380186080933, 1.9032155275344849, 0.06374312937259674, -0.22130979597568512, -0.44301819801330566, 0.5417097210884094, 0.7524277567863464, 0.5212240219116211, -2.1290597915649414, -0.5842500329017639, 3.0931198596954346, -0.11097808927297592, -0.14464539289474487, 0.5688159465789795, 0.16090941429138184, -0.21972547471523285, -0.4520130157470703, -1.1496108770370483, -0.8808988928794861, -1.2421499490737915, 0.1586376577615738, -1.8454872369766235, 1.487928867340088, 1.3282254934310913, -1.7269231081008911, 0.7680380344390869, 0.2537914514541626, 1.2496075630187988, 0.028239088132977486, 1.2847295999526978, 0.05384236201643944, -0.6889336109161377, -0.10214635729789734, 1.706203818321228, 0.7566632628440857, 1.2406073808670044, -0.6710389852523804, 0.714097261428833, 0.61263507604599, 1.731459617614746, 1.4848030805587769, -0.09957940876483917, 0.9980475902557373, -0.9143210649490356, 1.640804648399353, 0.21129192411899567, -1.3856806755065918, 0.08394630998373032, -0.0627734363079071, -0.8387811779975891, -1.274341106414795, -0.7607242465019226, -0.9043948650360107, -0.8940670490264893, 1.1757479906082153, -1.3523929119110107, -1.7623387575149536, -1.0507235527038574, 0.3503958582878113, 0.02661556750535965, -1.0567069053649902, 1.569257140159607, -2.5343825817108154, -1.1700160503387451, -2.6053402423858643, -0.22448813915252686, -0.7373654246330261, 0.9207747578620911, 1.3956713676452637, 3.2210757732391357, 1.6515047550201416, 0.33349788188934326, 0.06397814303636551, 1.7291120290756226, -1.094401240348816, 0.23663035035133362, 0.053465835750103, -1.2056467533111572, 0.6898333430290222, -0.06707268953323364, -0.9568884968757629, 0.5941333174705505, -1.476128339767456, -0.8777217268943787, 1.2364187240600586, 1.0980768203735352, -0.9279065132141113, -0.29835450649261475, 0.5645003318786621, -0.8380849361419678, 1.6428983211517334, -0.3586980700492859, 1.2480320930480957, 0.8381903171539307, -0.3804267644882202, 1.0291763544082642, -0.40415164828300476, 0.3528318405151367, -0.6995850801467896, -0.7198271155357361, -2.0334599018096924, 1.2833013534545898, 2.369290828704834, 1.152774453163147, -1.1525814533233643, 1.1216903924942017, -1.2126569747924805, 1.3912456035614014, 1.8666801452636719, 0.6225759983062744, 0.9737569689750671, -0.8551337122917175, -1.2611526250839233, -0.6561179161071777, 1.82291579246521, -0.37944623827934265, 0.7146246433258057, -1.6440200805664062, 0.06891746819019318, -1.1842716932296753, 0.379809707403183, 0.007537170313298702, -1.4256666898727417, -1.3669590950012207, 0.27350807189941406, 0.4684355854988098, 0.16905373334884644, -1.0946049690246582, 0.7056757807731628, 0.16601626574993134, 0.9384967684745789, -0.16147558391094208, 0.5260873436927795, 0.3767448663711548, 0.10383237898349762, 0.5318472981452942, -0.29207295179367065, -0.2833499610424042, 1.057372808456421, -1.0695862770080566, 0.0475822351872921, -1.3264158964157104, 0.8790808916091919, -0.4632934331893921, -0.40695810317993164, -0.8068366646766663, 1.6642227172851562, 0.040075045078992844, 0.35666990280151367, 0.17039138078689575, 1.3595571517944336, 0.2091066837310791, -1.790257453918457, -0.14204658567905426, -0.8942421078681946, -0.6430975198745728, 0.10981684178113937, -1.804883599281311, 1.1842763423919678, -1.24368417263031, -0.5358671545982361, -1.46255624294281, 0.5698097348213196, -0.5317828059196472, 0.977816104888916, 0.05215819552540779, -0.5557410717010498, 0.615699827671051, 0.6564844846725464, 0.6777907013893127, -1.696853756904602, -1.023772120475769, -1.1086416244506836, 0.6289114356040955, 0.013771651312708855, 0.25501978397369385, -1.6161153316497803, -0.526405394077301, -0.9471992254257202, -0.8376340270042419, -0.5907987952232361, -1.0480035543441772, 1.9593465328216553, 1.2426882982254028, 0.2542351186275482, -0.23145610094070435, 1.51851487159729, -0.25935882329940796, -2.1709110736846924, -0.8215835094451904, -0.6430199146270752, 0.3464706838130951, -0.6446176171302795, 1.2252380847930908, 0.45530641078948975, 1.1712491512298584, -0.18156151473522186, -0.7710192799568176, 0.5751693248748779, -0.39131057262420654, 0.30882978439331055, -0.7390562891960144, 0.6035337448120117, 0.8595813512802124, 0.15085357427597046, -0.7793768048286438, 1.0739483833312988, 1.791110634803772, 1.104411005973816, -0.8714396357536316, -0.5570812821388245, 2.570626735687256, 1.3024992942810059, 1.9460804462432861, 0.686896562576294, -0.774147629737854, -1.0016661882400513, -0.49778836965560913, -1.1689823865890503, -1.419446349143982, -0.42749708890914917, -0.9394029378890991, 0.058828599750995636, 0.6124578714370728, 0.24665343761444092, -0.29712238907814026, 0.693839967250824, -1.178518295288086, 0.0376446470618248, 0.6405397057533264, -0.1695755273103714, 0.8535405993461609, 0.6309576034545898, -0.8225051760673523, 0.40563637018203735, -0.9827402234077454, -0.735365092754364, 1.3901395797729492, -0.3514852821826935, 0.7664453387260437, 0.26631227135658264, 0.2398286610841751, 0.5896961092948914, -0.06729698181152344, -0.8071364164352417, -0.1917441487312317, -1.3351877927780151, -1.4601930379867554, -0.10187868028879166, 0.2765698730945587, 1.6569209098815918, 2.2996883392333984, -0.1734047830104828, 0.49320757389068604, 0.2851082682609558, -0.3324120342731476, 1.000276803970337, 0.640552818775177, -0.4949168860912323, 1.7357003688812256, 0.37376052141189575, 1.1286134719848633, 0.74790358543396, -0.49747660756111145, 0.38960185647010803, 0.7214725017547607, 0.2873740792274475, -1.8281148672103882, 1.5530651807785034, 0.2672337293624878, 0.7752021551132202, -0.7296086549758911, -1.0702122449874878, -0.7555668354034424, -0.07521167397499084, 0.17766767740249634, 0.1281154900789261, -0.5536224246025085, 1.2796679735183716, 3.7003204822540283, 0.44857895374298096, 1.8934590816497803, 1.4403653144836426, 0.31505730748176575, 0.08193549513816833, 1.8689464330673218, -0.6399283409118652, -0.1806354522705078, -1.5376900434494019, -0.5363141894340515, 1.6298351287841797, 0.6694757342338562, 0.4768959879875183, -0.9514791369438171, -0.6585602760314941, 1.0357097387313843, 0.1490219682455063, -1.1325511932373047, -0.7112839818000793, -0.37442687153816223, 0.5762094259262085, 0.5173296332359314, 0.42988452315330505, 0.9000969529151917, 0.18520960211753845, 0.3025689125061035, 0.02042355202138424, -0.8287293910980225, -1.166344165802002, -0.17104612290859222, 0.6310255527496338, 0.88917076587677, -0.4429313838481903, 0.23959878087043762, -0.441776305437088, -0.12205473333597183, 0.8638238310813904, -1.5548938512802124, -0.9095342755317688, -0.41395440697669983, -1.4900356531143188, -1.594130277633667, -0.5336827635765076, -0.20785635709762573, 0.626018226146698, 0.4386672377586365, -0.54985111951828, -0.2853519916534424, -0.49356117844581604, -0.8899523615837097, -0.726265549659729, 0.5792196989059448, 1.538196086883545, -2.3447964191436768, 0.32072558999061584, 1.6765164136886597, -1.7594166994094849, -1.263927698135376, 0.5634179711341858, -2.1160686016082764, -1.1589456796646118, 0.25693079829216003, -0.23546400666236877, -1.0461441278457642, -0.24014714360237122, 0.5925883054733276, -0.024775944650173187, 0.8094208836555481, -0.7540750503540039, -1.047071099281311, 0.16877032816410065, -1.5846126079559326, 0.16642290353775024, -0.12218711525201797, 0.14362843334674835, -0.018401794135570526, 0.08544988185167313, 1.1463905572891235, 2.7153937816619873, -0.4426812529563904, -0.26606592535972595, 0.3711167871952057, -0.26186326146125793, 0.5515829920768738, -0.09654287248849869, -0.8748362064361572, -0.9168392419815063, 1.436554193496704, 0.4098312258720398, 0.43822574615478516, -0.3912876546382904, 1.4692699909210205, 1.11492121219635, 0.2870624363422394, 0.8176161646842957, 1.4648714065551758, 0.9953866600990295, 0.3645622134208679, 1.3824020624160767, -1.1988763809204102, -2.0935182571411133, 1.0230746269226074, -0.27609583735466003, 0.4679693877696991, -1.4868965148925781, 1.814522385597229, -0.13079072535037994, -0.7625600099563599, 0.878150224685669, -0.3503188490867615, 0.18140444159507751, -0.7904288172721863, -0.9565463662147522, 0.3181121051311493, 0.3752124309539795, -0.2711433470249176, 0.35355809330940247, -0.48870396614074707, 0.436670184135437, -1.745173454284668, 0.4753681719303131, -0.82671058177948, 1.8080425262451172, 0.306943416595459, -0.0677223727107048, 1.1313084363937378, 0.5152454972267151, -0.9733234643936157, -0.6131827235221863, -0.6004475355148315, 0.5408378839492798, 0.8017586469650269, -2.1350409984588623, -0.5585073828697205, -0.9752540588378906, -0.41899287700653076, -0.18027812242507935, 1.0728427171707153, -0.4890928268432617, 2.1737632751464844, -2.381561517715454, 2.443427085876465, -0.3148861825466156, -0.8718628883361816, -1.023753046989441, 0.2931516468524933, 0.9298709034919739, -0.5011475682258606, -0.25621694326400757, 0.14190299808979034, -0.6030929088592529, 0.34117162227630615, -1.2068016529083252, 0.213005930185318, -0.600941002368927, -0.22122476994991302, -0.5010154247283936, 1.0603504180908203, -0.25741928815841675, 1.1701422929763794, -0.8605117797851562, -0.47906312346458435, -1.0622888803482056, 1.2135051488876343, 0.16492843627929688, 0.17835991084575653, 0.14446842670440674, 1.948081135749817, -1.3425531387329102, -2.6415510177612305, -0.23971538245677948, -0.3194997310638428, 0.6988815665245056, -0.47623956203460693, 0.2532622218132019, 0.044969238340854645, -1.1649699211120605, 0.6989763379096985, -0.9567526578903198, 0.34903761744499207, 0.2817600667476654, -0.3187333047389984, 0.13926076889038086, -0.12670862674713135, -0.2179601490497589, -0.151510551571846, -0.40449070930480957, 0.21949250996112823, -0.5113486647605896, -0.6361522078514099, -1.3459076881408691, -2.1079061031341553, -0.24797706305980682, -0.24507580697536469, -2.1979751586914062, 0.35394760966300964, -1.0224816799163818, -0.14223553240299225, 1.1406041383743286, 0.328610360622406, 0.9105868339538574, -1.9146134853363037, -2.338693618774414, -0.4764689803123474, 0.2538403570652008, -0.008946338668465614, -0.2789311110973358, 1.088496208190918, 0.39555051922798157, -1.682628870010376, -0.4200603663921356, 1.544319987297058, 0.44288551807403564, 1.8359155654907227, 0.28776103258132935, -0.6023311018943787, 0.2232286036014557, -1.9663145542144775, -0.03731134161353111, -1.0181772708892822, -1.2014306783676147, -0.7230145931243896, -2.2379565238952637, -0.27319610118865967, -1.105377197265625, 1.030988335609436, -2.3534398078918457, 0.1693745255470276, -1.276187777519226, -0.5187851190567017, 0.13431185483932495, -0.12762747704982758, 0.3836701810359955, -0.04320462793111801, 0.8080945014953613, 1.0102115869522095, 0.0737217590212822, -0.12901228666305542, 0.7561929225921631, 1.092960238456726, -0.5712215304374695, 1.9463825225830078, -0.575969934463501, -0.07178556174039841, -0.6147375106811523, -1.1678104400634766, -0.7951822876930237, 1.8461384773254395, -0.6068899631500244, -1.7682253122329712, -0.2913987934589386, 0.6856927871704102, -0.04872973635792732, 0.8739187717437744, -2.50120210647583, -2.259877920150757, 0.4572053551673889, 0.04297442361712456, 0.7414877414703369, -2.1006886959075928, 0.9531407356262207, 0.6489856839179993, -0.008561347611248493, 0.10421305149793625, -0.7636427879333496, -0.6429280042648315, 0.638858437538147, -0.5380927920341492, -0.6061597466468811, -0.9581748247146606, 0.777077317237854, 0.19099266827106476, 1.243065357208252, 1.5225999355316162, 2.0159924030303955, -0.5219894647598267, -0.16107779741287231, 0.8285912275314331, 1.0668236017227173, -0.29568642377853394, -1.8073171377182007, 1.3390662670135498, -2.2593441009521484, 2.7641360759735107, -0.19633974134922028, 0.593334436416626, 0.27697625756263733, -1.9250386953353882, -0.7635957598686218, -0.6317559480667114, 1.1668587923049927, -1.1722815036773682, -0.8704476952552795, 1.0761791467666626, -1.9931362867355347, -1.3811477422714233, -0.29083266854286194, -1.2042593955993652, 0.3844909369945526, 3.165809392929077, 0.286406546831131, 0.9150902032852173, 1.276802659034729, -1.4971905946731567, 0.38335925340652466, 1.51503324508667, -0.5063571333885193, -0.7247360944747925, -2.1897315979003906, -0.5735135674476624, -2.7052156925201416, -0.7446706891059875, 0.7483363747596741, -0.5386381149291992, 0.7115646004676819, -0.5715804696083069, 0.5901307463645935, 0.5995638370513916, 0.1978803277015686, 1.8083804845809937, -1.0081067085266113, 0.5636524558067322, 0.028411589562892914, 0.06305092573165894, 0.04341411963105202, -1.1253435611724854, 0.45127201080322266, -0.17576240003108978, 0.8146858811378479, -1.7267006635665894, 0.8630567193031311, -0.063905268907547, 1.7714941501617432, -2.131547451019287, 0.5532304048538208, -0.2725856900215149, 0.6100450158119202, -0.3507147431373596, -0.8395338654518127, -1.5820777416229248, -0.516194224357605, 1.0278098583221436, 0.32522955536842346, 1.7585623264312744, 0.23083940148353577, 1.4311805963516235, -0.6719446182250977, -0.5353761315345764, 0.24874603748321533, 0.21964748203754425, -0.18824298679828644, -0.8340126872062683, -0.21712170541286469, -0.8732687830924988, 0.6778738498687744, -1.6212875843048096, 1.5926638841629028, -0.2008974701166153, -0.5121175646781921, -0.9704321026802063, 0.20501923561096191, 0.32824841141700745, -0.21027322113513947, -0.3521779179573059, 0.31099995970726013, -0.19547881186008453, -1.1687133312225342, -1.2426111698150635, 0.9771267175674438, 0.674841582775116, 0.5892694592475891, 0.448659747838974, -0.4589981734752655, -0.009619868360459805, -0.7399412393569946, -0.34762197732925415, -0.39168858528137207, 0.015702657401561737, -0.2879708409309387, 0.4440512955188751, -0.13560999929904938, 0.7235590815544128, 0.08344601094722748, 0.1256464570760727, -0.6602649092674255, 0.8457322120666504, 0.5861542820930481, 1.257689118385315, 0.6604192852973938, 0.5027210116386414, 1.0651954412460327, 0.3043057918548584, -0.5882224440574646, -0.052464645355939865, -0.1581266224384308, -0.5411536693572998, 0.6917202472686768, -2.110179901123047, -0.6843515634536743, 0.4074980616569519, -0.7693015336990356, -0.5695151686668396, 0.6966409683227539, 0.382631778717041, 0.40282005071640015, 0.07313037663698196, -0.5071556568145752, -0.5981404781341553, -0.40656283497810364, -0.4380097985267639, 0.15797437727451324, 0.5434906482696533, 0.4366644024848938, 0.21203508973121643, 0.07586782425642014, 0.5101302862167358, 0.9765737056732178, 0.6644192337989807, 0.1195206567645073, -0.9290691614151001, -0.6689575910568237, 0.6061702966690063, 1.5673092603683472, -0.273931086063385, 0.14368489384651184, 1.1035161018371582, -0.9057912826538086, -0.36753976345062256, 0.03809541463851929, -0.3035763204097748, 0.21782700717449188, 0.3605724573135376, -1.8106948137283325, 1.5785070657730103, -0.7860183715820312, 0.27027326822280884, -0.8882490992546082, -1.6479371786117554, -0.17505024373531342, -1.400072455406189, 0.35526400804519653, 1.2877649068832397, -1.6211615800857544, 0.3782489001750946, 0.1862797737121582, 0.008772071450948715, 0.8881219625473022, 0.39030537009239197, 0.4352158010005951, -1.5093462467193604, 1.815412163734436, -1.6853702068328857, -0.7493025660514832, -0.9291269183158875, -0.9587193131446838, 0.5507558584213257, 1.4436657428741455, 0.5977384448051453, -0.6054054498672485, 0.06751081347465515, 0.37731030583381653, -0.31092900037765503, -0.9481405019760132, -0.663902997970581, 0.6490958333015442, 1.1098814010620117, 1.2027289867401123, 1.2034862041473389, -0.6044384837150574, 0.4102493226528168, -0.8225919604301453, -0.445037305355072, 0.8033137917518616, -0.06003906577825546, -0.396605521440506, -2.1667745113372803, 0.5569685101509094, 0.9366399645805359, 0.3774924874305725, -1.279451608657837, 0.5654723644256592, 0.7902286052703857, -0.5774267315864563, 0.39510300755500793, 1.026453971862793, 0.979279100894928, -0.8277481198310852, 0.7099658846855164, -0.25538167357444763, 0.8862530589103699, 0.28930437564849854, 0.8004916906356812, -1.190102219581604, 1.371705412864685, -0.04803692176938057, -0.6982306241989136, -0.763679563999176, -0.755526602268219, -0.2896100878715515, 1.3965086936950684, 1.6455137729644775, -1.935030221939087, 0.03129461780190468, -1.0845062732696533, -1.2765432596206665, 0.9287158846855164, 0.5768962502479553, 0.9864458441734314, -0.9319474101066589, -1.0014009475708008, 0.4859902560710907, 1.057206153869629, 1.577593207359314, 0.9586502313613892, -1.2809053659439087, 0.71505206823349, 0.5780746340751648, -0.19962789118289948, -1.200082540512085, -2.0751795768737793, -0.7772534489631653, -1.0047791004180908, 1.4499773979187012, 1.3085765838623047, -0.17713980376720428, 0.47944027185440063, 1.4988113641738892, 0.00714547373354435, 0.5032594203948975, -1.1087660789489746, 0.43263110518455505, -0.7128053307533264, 1.3311994075775146, -1.0131118297576904, -0.10082750767469406, -0.7637353539466858, -0.2674446702003479], [3.2318243980407715, -1.571420669555664, -0.66796875, -1.7568970918655396, -0.23232875764369965, -1.8457940816879272, -0.393268346786499, -1.061705470085144, 0.8988579511642456, 0.09408445656299591, 1.0033010244369507, 0.023274322971701622, -0.046394336968660355, 0.07091929018497467, -0.31495019793510437, -1.6525737047195435, -0.7966261506080627, -0.9650616645812988, -0.550945520401001, 0.3373675048351288, -1.2707866430282593, 0.6068507432937622, -0.8225364089012146, -1.9510207176208496, -1.64216947555542, -1.601706862449646, 0.3036738634109497, 0.6760296821594238, 1.1405317783355713, -0.6438208818435669, -0.6374109983444214, -0.5490251183509827, 1.4099146127700806, -1.7411963939666748, 0.5367017984390259, 2.165214776992798, -0.43397796154022217, 0.8929879069328308, 0.7882238626480103, -0.158933624625206, -0.1516256332397461, -0.25098690390586853, 0.6615813970565796, 1.365421175956726, -0.990837037563324, -1.0680309534072876, -0.684769332408905, 0.6445358395576477, -1.8661985397338867, 0.965395450592041, 1.030380368232727, -0.41208940744400024, 0.34853339195251465, 0.4659469425678253, -1.2994334697723389, 0.4036746919155121, -0.29323598742485046, -1.625131607055664, -0.3246254324913025, 0.531342089176178, 0.32764339447021484, -0.8317323327064514, 0.5736133456230164, -1.3344006538391113, 1.1810063123703003, -1.2308663129806519, -0.7517420649528503, -0.200897216796875, 1.0190271139144897, 0.271576851606369, 0.2880275845527649, 0.25568440556526184, 0.2519000768661499, 1.0981131792068481, -0.1698089987039566, -0.30556172132492065, 0.49472424387931824, 1.0057976245880127, -0.11478215456008911, -0.10742290318012238, -0.5387670993804932, 1.281569242477417, 0.1556015908718109, 1.56065833568573, -1.26714026927948, 1.245385766029358, 2.1701321601867676, 1.5571070909500122, -0.2671416997909546, 0.19244521856307983, -0.8022152781486511, 0.449801504611969, -1.545102596282959, 0.8911282420158386, -0.8701595067977905, 1.7702444791793823, -1.9323740005493164, -0.2244655191898346, 0.9965839385986328, -0.1697264015674591, -1.4853895902633667, 0.9433243274688721, 0.49121469259262085, 1.1418734788894653, -0.7031925320625305, -1.6138527393341064, -0.19816045463085175, 0.8751148581504822, 1.154667854309082, 0.5599768161773682, 0.15623633563518524, -1.9301095008850098, -1.511877179145813, -0.7095443606376648, -1.4168826341629028, -0.33738815784454346, 0.25063619017601013, -0.5257834792137146, -2.3202931880950928, 0.5618410110473633, -0.9795958399772644, 1.3976832628250122, -0.3532901406288147, -0.09384679794311523, -0.29542210698127747, 1.3591049909591675, 0.3595210909843445, 0.02782467193901539, -0.03274691849946976, 0.3567332327365875, 2.11615252494812, 0.2677406668663025, -1.302579402923584, 2.3410451412200928, 0.8431873917579651, 0.7135118842124939, -0.3863579034805298, -0.31930145621299744, -1.1712740659713745, 1.438905954360962, -0.07903178036212921, -0.5988264083862305, -0.8245305418968201, -0.15868274867534637, -0.28993406891822815, -1.2103394269943237, -1.5822302103042603, -0.4824027121067047, 1.2481566667556763, -0.08232080936431885, 0.9935387969017029, 0.5839124917984009, -0.694126546382904, -0.6300450563430786, 2.7864158153533936, -1.6487090587615967, 0.740870475769043, -1.1720067262649536, 3.2947661876678467, -0.4894632399082184, -0.5887594819068909, -0.13069932162761688, 1.2701400518417358, 0.41151827573776245, -1.8977851867675781, -0.4860999882221222, -1.127407193183899, -0.04410053417086601, -0.3292778432369232, 0.12472673505544662, 1.066986083984375, 0.41569575667381287, 1.4663163423538208, -1.1564470529556274, -0.6531933546066284, 0.4876375198364258, 1.1374313831329346, 1.195814609527588, -0.6253374218940735, 1.2517179250717163, -0.8943113088607788, 1.5521844625473022, 0.0030842041596770287, 1.6347912549972534, -0.8203395009040833, -0.3553365468978882, -1.2033449411392212, -0.470375657081604, 0.7816309928894043, -1.2921154499053955, -1.7192126512527466, 0.014647521078586578, 0.13675758242607117, 0.23340128362178802, 1.1955219507217407, -0.8824946880340576, -0.442905992269516, -1.8997223377227783, 0.40661245584487915, -0.22876785695552826, -1.2151964902877808, 0.664297342300415, 1.5503615140914917, -0.8338943719863892, 0.2604438066482544, 0.1968766748905182, -1.2944090366363525, 0.12340867519378662, -1.4254640340805054, -0.24794138967990875, -0.567984402179718, -1.5786911249160767, 1.1138155460357666, 0.39903953671455383, -0.10669435560703278, -0.35384002327919006, -1.3385826349258423, 0.8357363939285278, -1.3975192308425903, -0.3133029341697693, -1.2996927499771118, -0.3348188102245331, 0.37388876080513, 0.32438868284225464, -0.7229135632514954, 0.1766207218170166, 0.5371943116188049, 0.06182875111699104, 1.6126638650894165, -1.2964104413986206, 0.8704490065574646, 1.658646821975708, -1.1583566665649414, -1.7215806245803833, -1.5823554992675781, -0.6153905987739563, 0.7665262818336487, 1.2102893590927124, -0.065760537981987, -2.0605990886688232, 0.15817227959632874, -0.37630152702331543, 0.43556368350982666, -0.2682015895843506, 1.893376111984253, 0.5716561079025269, -0.410078227519989, -1.5475503206253052, 1.6478214263916016, -0.011200882494449615, -0.6853603720664978, -1.2882596254348755, 0.8570297956466675, 0.6638312935829163, -2.1790578365325928, -0.6841384172439575, 0.03960868716239929, 0.8649336099624634, 1.4213944673538208, 0.8718456625938416, 0.7528297901153564, 0.6311647891998291, 0.8495796322822571, -2.5586111545562744, 0.2641168236732483, 2.3323419094085693, 0.758497416973114, 2.9640116691589355, 0.7467411756515503, -0.5198524594306946, -0.45557519793510437, -0.48285847902297974, -1.2923396825790405, -1.2194448709487915, -0.8282161951065063, -0.174539715051651, -0.7619698643684387, -0.650475025177002, -0.9214239120483398, 1.3289210796356201, -0.2577533721923828, -0.48544585704803467, -1.0125573873519897, -1.1096305847167969, 0.4296027421951294, -0.13160593807697296, 1.0871829986572266, 0.9188790321350098, 1.7790985107421875, 0.2724166214466095, -0.5312631726264954, -0.37147822976112366, 0.6551902294158936, 1.4360716342926025, -0.9441871643066406, -1.5452626943588257, -0.9135655760765076, -2.950579881668091, 1.1786376237869263, 0.758648693561554, 0.14429187774658203, 1.5010570287704468, -0.9406653642654419, -0.25887078046798706, 1.402146339416504, -1.2007559537887573, 0.7229631543159485, 0.7574353218078613, 0.19541305303573608, -1.049918532371521, 0.3619184195995331, 1.0825918912887573, 2.4651846885681152, 1.6383146047592163, -0.0053514097817242146, 0.16617144644260406, -0.8550509810447693, 1.2031031847000122, 0.6028315424919128, -0.20902426540851593, -1.80242919921875, 0.5363485217094421, 0.39234980940818787, -0.17635981738567352, -1.2629565000534058, 1.3546781539916992, -0.05589427426457405, 0.3930876553058624, -1.0004018545150757, -0.23261164128780365, -0.05116508528590202, 1.3074870109558105, -0.09867369383573532, 1.9123152494430542, 0.09558367729187012, -0.9813070297241211, 0.5864716172218323, 0.043857596814632416, -1.0182664394378662, 0.7557359933853149, -2.7072832584381104, -1.2420954704284668, 0.7183786630630493, -0.275448203086853, 1.9678908586502075, -0.21910306811332703, 0.5615567564964294, -0.8757466673851013, -0.002806817414239049, 0.21000327169895172, 0.629083514213562, -1.4629924297332764, 1.3467602729797363, 0.8832266330718994, 0.611397922039032, 0.6116701364517212, -0.9042441844940186, -0.15589891374111176, -0.5066195130348206, -0.29179897904396057, -1.4461171627044678, -1.6626604795455933, 1.3768256902694702, -0.41498470306396484, -1.094459056854248, -0.6335212588310242, -0.1861506998538971, 0.04026108235120773, 0.3468596339225769, 2.081007719039917, 1.619174599647522, 0.38793352246284485, -2.1918301582336426, 0.14896678924560547, -1.8065118789672852, -0.272122323513031, -1.4049804210662842, -1.5427638292312622, -0.23974451422691345, -2.023467779159546, 1.6555614471435547, -0.5583117008209229, -0.15030844509601593, 0.917437732219696, 1.6435409784317017, -1.1428098678588867, 1.2201627492904663, 0.0544167198240757, 0.1750907152891159, 0.24985074996948242, 1.3832149505615234, -0.1382286697626114, -1.000524640083313, -0.9283040165901184, 1.0562776327133179, -0.5055744051933289, 0.7069079875946045, -0.7048068046569824, -0.3760146200656891, 0.6187673807144165, 0.35645127296447754, -2.2621469497680664, -1.931886076927185, -2.6658263206481934, 0.17151330411434174, -0.6901033520698547, 0.17672832310199738, 0.393346905708313, 0.8699092268943787, 0.49961546063423157, 0.9312201738357544, -0.9372875690460205, 0.1543266326189041, -0.8514562845230103, 1.2087383270263672, 0.86522376537323, -1.2580742835998535, 1.0630658864974976, -0.4877554774284363, 1.46017324924469, 0.2973569929599762, 1.4577906131744385, -0.6703738570213318, -0.6830969452857971, 1.33597993850708, 0.5798009037971497, -0.9433825016021729, 0.12839795649051666, 1.377023696899414, 2.085977792739868, -1.5780502557754517, -2.2487058639526367, 0.8897404670715332, 1.238854169845581, -0.8695436716079712, 1.0894991159439087, 0.6038877367973328, 0.5496521592140198, 1.4865325689315796, 1.4184437990188599, -0.9657285809516907, -0.9261980652809143, 1.3731880187988281, 2.0402050018310547, 1.0281533002853394, -1.0331616401672363, 0.25664156675338745, 1.782544493675232, 0.7835038304328918, -0.6958780288696289, 0.5265657305717468, 0.21258391439914703, -0.6251655220985413, -2.2804901599884033, -0.6883041858673096, -0.07168653607368469, 0.4352372884750366, 1.6054017543792725, 2.0520927906036377, 0.774833619594574, 0.6709729433059692, 0.5407094955444336, -0.1742473691701889, 0.29772135615348816, -0.5942434668540955, 2.1698427200317383, 1.4407575130462646, 0.7158889174461365, -1.1396706104278564, -0.20256295800209045, 1.7329020500183105, 0.24884247779846191, 0.6494420766830444, 0.4453432857990265, -0.3126964569091797, 0.467199444770813, 0.40715521574020386, -2.0331084728240967, 1.8094631433486938, -0.23318368196487427, 1.101240634918213, 0.5276790261268616, 0.0727144107222557, 0.01287276204675436, -0.5617552995681763, -0.2214358150959015, -1.7959020137786865, -0.3376171290874481, -1.8192862272262573, -0.6671276092529297, 0.8420031666755676, 0.6873385906219482, -1.0228904485702515, -0.47174572944641113, -0.26795902848243713, 0.07977856695652008, -0.6489810347557068, 0.5049585103988647, -0.4791170656681061, -0.6478888988494873, 0.14693932235240936, -0.21356350183486938, -0.44000816345214844, 1.5421661138534546, -0.25524595379829407, -1.2729883193969727, 0.16261978447437286, -0.4866374135017395, -1.0535240173339844, 0.18506453931331635, 0.022944509983062744, 0.21670092642307281, -1.5992966890335083, 0.8010725378990173, 0.26061388850212097, -0.11770205944776535, -1.0875149965286255, -1.2532122135162354, 1.512641191482544, 2.212059736251831, 0.16768445074558258, -1.4364348649978638, -0.49877962470054626, 0.6799737811088562, -0.0390399694442749, 0.12097223103046417, -0.06334729492664337, 1.4035707712173462, -0.022978629916906357, 1.1614990234375, 1.3497848510742188, -0.07143071293830872, 0.14691203832626343, -0.8821695446968079, 0.08610015362501144, -1.0360698699951172, 1.0103565454483032, 0.09819898754358292, -1.353683352470398, 0.8234131336212158, 1.3172396421432495, -2.096252679824829, 0.3468065857887268, -0.29357001185417175, -0.3369198143482208, -0.2068597823381424, 0.4845588505268097, 0.2739969789981842, -0.5877981185913086, 0.5277609825134277, -0.7566215991973877, 0.7840121388435364, 0.6833003163337708, 0.8940929770469666, 0.8276481628417969, 1.08175528049469, 0.46855345368385315, 0.5704155564308167, 2.2095742225646973, 0.033641744405031204, 1.6370738744735718, -0.20683975517749786, -0.23949995636940002, -0.19745439291000366, -0.6770710349082947, -0.1379711925983429, 1.6926157474517822, -0.4274837374687195, -0.6464881300926208, 0.9922600388526917, 0.7402392625808716, 0.892556369304657, -0.176565021276474, -0.8530378937721252, -1.5189244747161865, 1.6168662309646606, 0.3399684429168701, -1.2359216213226318, 0.5824561715126038, -0.8527070879936218, 0.03499147668480873, -0.09164831042289734, -1.2743483781814575, -0.3649672269821167, -0.3954917788505554, -0.6908541917800903, -0.918251633644104, -0.7548859715461731, -0.9529845118522644, 1.6418735980987549, 0.15301942825317383, 0.9639360904693604, -0.8289994597434998, -0.9620170593261719, -0.965385377407074, 0.4975278675556183, -1.6230568885803223, 0.5729446411132812, 0.8007673025131226, 0.29035094380378723, 1.5902292728424072, 0.835106372833252, 1.2001451253890991, 0.31854385137557983, -0.13309824466705322, -1.0913591384887695, 0.45737919211387634, -0.04475238919258118, 0.509264349937439, -0.46127861738204956, -1.580350399017334, -1.6303693056106567, -0.6924476623535156, -0.03745852783322334, 0.6274394392967224, 0.5232451558113098, 1.2158331871032715, 0.4836580157279968, -0.5483875870704651, -0.23708872497081757, -0.7909340262413025, 0.6021624207496643, -0.5117174386978149, -0.6573287844657898, -1.0638118982315063, 0.9579866528511047, 0.007510545197874308, -0.02489793673157692, 0.08686564862728119, 0.9578282237052917, 1.2935906648635864, 0.6405186057090759, -1.0721468925476074, -0.8346598744392395, 0.14052477478981018, -1.500174880027771, 0.3440350294113159, -0.4235440790653229, -0.25060757994651794, 0.17932307720184326, -0.015098056755959988, 0.4015691876411438, 0.8024119734764099, -0.2737017869949341, 0.5884144306182861, 1.797930121421814, 0.0495181605219841, 0.5869539380073547, 0.8449068069458008, 2.6148016452789307, 0.2216188609600067, 0.9083101153373718, 0.6899290680885315, 1.2448745965957642, -0.5201125741004944, 0.12524011731147766, 1.1983507871627808, 0.4755170941352844, -0.6651802062988281, -0.5860357284545898, 0.02449088543653488, -1.8761093616485596, -1.0365808010101318, -0.9779237508773804, -0.6293905377388, 0.8035164475440979, -0.20000320672988892, -0.11899566650390625, -0.4943728446960449, 1.2159909009933472, -1.0194108486175537, 0.8074684143066406, -0.5085784792900085, -2.486523389816284, -2.153907060623169, -1.098727822303772, 3.7408595085144043, -0.4993986487388611, -1.0114803314208984, 0.28454825282096863, -0.7714453935623169, -0.09859026968479156, -0.40879690647125244, -0.36644941568374634, -1.2928227186203003, 1.3543689250946045, 0.9799236059188843, 0.6490756273269653, -0.8177474141120911, 0.41879165172576904, -0.3553895056247711, 0.3635541796684265, -0.6958770751953125, -3.222294330596924, -0.34579890966415405, 1.1925214529037476, -1.7575304508209229, 0.47676882147789, -0.47552424669265747, 1.1299070119857788, 1.5023220777511597, 0.5796078443527222, -2.5272774696350098, -0.31214582920074463, -0.4376281797885895, -1.1235895156860352, 0.6906811594963074, -0.5918244123458862, 1.4062951803207397, -0.5170115828514099, -0.06563819199800491, -0.2598462402820587, 0.437375545501709, -0.5374777913093567, 2.6778926849365234, 0.514273464679718, -0.6387911438941956, 1.1916015148162842, 0.5317872166633606, -1.4398884773254395, 0.1333523690700531, 0.10602818429470062, 0.1765948385000229, 1.4619295597076416, -1.179291844367981, -1.2729473114013672, -0.6160606741905212, 0.03197126463055611, -1.0378271341323853, -0.06370898336172104, 0.5337523818016052, -0.004079421050846577, -0.6142856478691101, 1.7713539600372314, 1.9040536880493164, -0.5626758933067322, -0.8779782652854919, -0.6668151617050171, -1.3798316717147827, 0.8070792555809021, -1.013767123222351, 0.6425756216049194, 0.3934638500213623, -0.5002721548080444, -0.40228068828582764, 1.5506781339645386, 1.7527514696121216, 0.5569964051246643, -1.3165031671524048, -1.4048287868499756, 0.7197438478469849, 0.09765221923589706, -0.005857095587998629, -0.37708228826522827, -1.5005232095718384, -2.696977138519287, -0.12926806509494781, 0.39437755942344666, 1.5018256902694702, 0.026207758113741875, -0.556267261505127, -0.42130130529403687, 0.10630647838115692, 0.6692371964454651, -1.316925287246704, 1.509910225868225, -1.2380353212356567, 0.3993239104747772, -0.30840566754341125, -0.9302687048912048, 1.5244139432907104, 1.0106796026229858, -1.2634397745132446, 0.35244864225387573, -0.49998393654823303, -0.943391740322113, -0.5135595798492432, -1.2723180055618286, -0.3935413062572479, -1.2929425239562988, 0.9288614988327026, 1.5862032175064087, 0.7742273807525635, 1.1107630729675293, -0.4392320215702057, 0.3979801535606384, -1.3415460586547852, -1.030755639076233, -0.518355667591095, -1.514981985092163, 0.7540362477302551, -1.5465874671936035, 0.6837502717971802, -1.4502865076065063, 1.1616499423980713, 0.5971282720565796, -0.6457124352455139, -1.082899570465088, 0.8049929141998291, 0.16684462130069733, -1.0720221996307373, 1.3066349029541016, 1.3482798337936401, 0.34645408391952515, 1.1244778633117676, -0.9613813757896423, 0.49264267086982727, -0.5888285636901855, 1.009993314743042, -0.4506896138191223, 1.6113461256027222, -0.8698092699050903, -0.6365585327148438, 0.2401205450296402, 1.0885056257247925, 0.498838871717453, -1.233751893043518, 0.7169530987739563, 0.664518415927887, -1.6334493160247803, -0.8616570234298706, 0.9705204963684082, -3.119405508041382, 1.6266499757766724, 0.4377932548522949, 0.952062726020813, -1.8780720233917236, -0.20658203959465027, 0.25932255387306213, 0.043815795332193375, -0.281995952129364, -0.5632833242416382, -1.205346941947937, -0.014036798849701881, 0.42344653606414795, -0.7519652843475342, -0.17506840825080872, -0.40517082810401917, 0.2523675560951233, 1.4360569715499878, -0.35296395421028137, 0.25554847717285156, -0.6940454840660095, 0.011297207325696945, 0.10651268064975739, -0.5618561506271362, 0.05469762906432152, 0.9662326574325562, -1.5988383293151855, -0.380545973777771, -0.6885274052619934, 1.6056832075119019, -1.945877194404602, -0.7071501016616821, -0.22316908836364746, -0.339245080947876, 1.0254799127578735, -1.1962863206863403, 1.1012276411056519, 1.2381680011749268, 0.9283692240715027, 0.2912765443325043, -0.5529453158378601, 0.4574166536331177, 2.4654743671417236, -0.34713712334632874, 0.6117624044418335, -1.6645597219467163, -0.9951775670051575, -0.22214806079864502, 1.0227148532867432, -1.4726999998092651, -2.014122724533081, 0.43010979890823364, -1.5905628204345703, -0.5080235004425049, -0.5420068502426147, 0.1324130892753601, 1.27354896068573, -1.0633171796798706, 0.6982432007789612, -2.02889084815979, 0.0324704572558403, -0.6142107844352722, 1.0110572576522827, 0.9326012134552002, 2.46585750579834, -0.6481149792671204, 1.1620099544525146, -0.17505480349063873, -0.8182035684585571, -0.21999500691890717, -0.8581497669219971, -1.7874962091445923, 0.4627580940723419, 1.1696295738220215, 1.0035377740859985, -0.06441835314035416, 0.352873831987381, 1.226327657699585, 1.069159984588623, -1.6749157905578613, 0.7668128609657288, 1.1565189361572266, 1.0451438426971436, 1.9024064540863037, 2.884695291519165, -0.42350178956985474, 1.5495281219482422, 0.37637972831726074, 1.244242548942566, -0.1590610295534134, -1.5284408330917358, 1.2829716205596924, 0.45585376024246216, 0.5741718411445618, 1.03364896774292, 2.0922322273254395, -1.1286901235580444, 0.7897312045097351, -0.15971355140209198, 1.3847391605377197, -0.21237589418888092, 0.596969723701477, 0.4742494523525238, -2.5443434715270996, 0.999855637550354, 1.3264771699905396, 0.8810970187187195, 0.2042248398065567, 0.8209429979324341, 0.5642221570014954, 1.1478878259658813, 0.10932724177837372, -0.005670202895998955, -1.11670982837677, -0.24192579090595245, -0.01407268363982439, 0.5133596062660217, 1.3112986087799072, -0.38889774680137634, 0.8786706328392029, 0.7687339186668396, 0.4829844832420349, 0.008200405165553093, -0.8625937700271606, 0.5580305457115173, -0.16667869687080383, -0.3058215081691742, 0.09812859445810318, 0.7915946841239929, 2.532961368560791, -0.016241081058979034, -0.19557084143161774, 0.9311347007751465, -0.6557808518409729, 0.9774234294891357, 0.31501245498657227, -1.6222323179244995, 0.13755586743354797, 0.48732292652130127, 0.454819917678833, 1.2614940404891968, -0.4852564036846161, -0.2532045245170593, 0.3731847107410431, -1.0545722246170044, -1.003927230834961, -1.0316901206970215, -0.8981037139892578, -0.24753646552562714, -0.0011152956867590547, -1.0631380081176758, 0.4731851816177368, 0.4384644627571106, 0.22384676337242126, 0.10361584275960922, -1.542184829711914, 0.6024917960166931, -1.291495680809021, 0.6207184791564941, -0.08817370980978012, 1.5716798305511475, -0.32598742842674255, 0.3461849093437195, 0.22086742520332336, 0.005823791958391666, -0.00768094789236784, 0.7591032981872559, 0.6688249707221985, -0.4244000017642975, -0.11073629558086395, -0.8986803293228149, -1.4150333404541016, -0.3590220808982849, -1.6245415210723877, -0.9706437587738037, -0.1995583027601242, 0.7583444714546204, 1.3447726964950562, -1.6007198095321655, 0.008069176226854324, -0.797588050365448, 1.2740347385406494, 0.4424535632133484, 1.4896025657653809, -0.9625552296638489, -0.6972742676734924, -0.0702965259552002, 0.15018011629581451, 1.0787380933761597, -0.7217448949813843, -0.2114313244819641, 0.5312010049819946, -1.0290919542312622, -0.9931401014328003, -0.3192461431026459, 1.3520811796188354, 0.2972738742828369, 0.31846314668655396, -0.8530130982398987, 0.042400430887937546, 0.8068479895591736, -0.5726598501205444], [0.831579864025116, 1.3627121448516846, 0.942663311958313, -0.2220056802034378, 1.3965392112731934, -0.013849682174623013, 0.3422946333885193, 0.9549143314361572, -0.33029136061668396, 0.8420743942260742, -0.8927963376045227, -0.26242056488990784, 1.4382059574127197, -1.097213625907898, 0.05538514256477356, -0.6171899437904358, 1.1830815076828003, -0.24602742493152618, 0.1431339681148529, -0.5694067478179932, -0.1430804282426834, 1.6109527349472046, -0.6001324653625488, 0.6468533873558044, -1.225472331047058, 0.4951149821281433, 0.2244209498167038, 1.2955818176269531, -2.281564950942993, -1.982950210571289, -1.1261804103851318, -0.30350953340530396, 1.0678566694259644, 0.35937201976776123, 1.2328581809997559, -0.04943742975592613, -1.4553470611572266, -0.06440272927284241, -0.26567280292510986, 0.1762728989124298, -0.11919119209051132, 0.3121922016143799, 0.6083278059959412, -1.192017912864685, 0.7501687407493591, -0.6599767804145813, 1.1014668941497803, 0.057821813970804214, 1.123177409172058, -1.6583279371261597, -1.3193461894989014, -0.5966565608978271, 0.2601834833621979, -0.5197452306747437, 1.0504120588302612, -0.8345900177955627, 2.2725486755371094, 0.6247511506080627, 0.6420059204101562, -0.07288620620965958, 0.07341578602790833, -1.1917117834091187, -0.9758883714675903, -0.04698004573583603, 0.8653217554092407, -0.6068873405456543, 0.20789912343025208, -1.521419644355774, -0.058665789663791656, -0.22817738354206085, 1.345884084701538, -0.6127129793167114, 0.8071817755699158, -0.4109663963317871, 1.373122215270996, -0.5196496844291687, 0.9231179356575012, 1.1880402565002441, 0.41913121938705444, -0.0011034802300855517, -1.7453672885894775, -1.3463363647460938, 0.9882275462150574, -0.4137112498283386, 1.5160380601882935, 1.76487135887146, -2.590275526046753, 0.7270005941390991, 0.03209463134407997, -0.5254538655281067, 2.3039915561676025, -0.8534089922904968, 1.5692538022994995, 0.11221596598625183, 0.5704431533813477, -0.6295212507247925, -1.4891886711120605, 0.34212106466293335, 1.5640983581542969, -0.9736828804016113, -1.1513400077819824, -1.0977693796157837, -0.7011387348175049, -1.9032320976257324, -0.12401889264583588, 0.20512403547763824, 1.4696964025497437, 0.5978777408599854, 2.07698655128479, 1.8099689483642578, 0.8291511535644531, 0.84245365858078, 2.26944899559021, 2.169983148574829, 0.57313472032547, 0.25655636191368103, -0.03642250597476959, 0.1382589042186737, -1.8166927099227905, 0.3401317000389099, -0.25052788853645325, -0.48523950576782227, -0.46610042452812195, -0.4808681607246399, 0.8651325106620789, -0.14716090261936188, -0.5649617910385132, -0.5385427474975586, 0.7463454604148865, 0.2339104861021042, 1.1762590408325195, -0.027128081768751144, 0.4258492588996887, -0.3422471284866333, -0.1497962474822998, -0.5602947473526001, -1.2955281734466553, 0.20586641132831573, -0.6196977496147156, 0.15673400461673737, -1.43110990524292, -0.7303026914596558, 1.306868553161621, 0.3108997344970703, -0.05690706893801689, -0.632951557636261, -1.2066924571990967, -1.2326031923294067, -2.2566962242126465, 1.756697416305542, 0.003656476503238082, -0.8070653676986694, -0.3034111559391022, -0.28857362270355225, -0.6862672567367554, -0.8563815355300903, 0.024792933836579323, 0.22395794093608856, -0.03213691711425781, 0.7975729703903198, -1.7649683952331543, -0.4055415689945221, 0.1605852097272873, 0.7456718683242798, 1.2807493209838867, -0.5468108057975769, 0.3219392001628876, 0.6242156028747559, 1.343880295753479, 1.5874230861663818, -0.1520010232925415, 0.6512452363967896, 0.21989724040031433, 0.4136241674423218, -0.5954620242118835, 0.6049226522445679, 0.44773778319358826, -2.067416191101074, 0.26453375816345215, 0.14085093140602112, 0.777722179889679, 0.5345015525817871, 1.2554484605789185, 0.6477760076522827, 2.4820995330810547, -0.2949639856815338, -0.3882829248905182, -0.1485653817653656, 0.16302131116390228, 0.0782100260257721, 0.4699486196041107, 1.2012348175048828, 0.6068161725997925, -1.8937718868255615, -0.9304455518722534, -1.0817335844039917, -0.9600414037704468, 0.23934543132781982, -0.49774983525276184, -1.8304357528686523, -1.3994629383087158, -0.26900455355644226, 0.010880643501877785, -0.9769380688667297, 0.9795283079147339, 1.079969048500061, 0.8199833035469055, -1.3402881622314453, -0.9028005003929138, -0.5436519980430603, 0.3238235414028168, -1.0932979583740234, 1.897708535194397, 0.2694292664527893, 0.8233237862586975, 0.23750510811805725, 0.8563818335533142, 0.15679620206356049, -1.350766897201538, -0.6388546824455261, 0.6285029053688049, -1.3616070747375488, 2.1394076347351074, -0.4070057272911072, 0.5667550563812256, -0.9885363578796387, -1.4572439193725586, 0.7245492339134216, -1.1469405889511108, -0.8787551522254944, -1.2822399139404297, -1.7868820428848267, 0.04366661235690117, -0.7198778390884399, 0.5184403657913208, 1.7206382751464844, 0.29529932141304016, 0.6051832437515259, -0.22656717896461487, 0.13689018785953522, -0.14024832844734192, 0.9094542264938354, -1.6215429306030273, -1.6055097579956055, -0.35362038016319275, -0.8219423294067383, -0.6790686845779419, -1.3672469854354858, 1.9702244997024536, -1.2034895420074463, 0.43777745962142944, 0.7764548063278198, 0.8781023025512695, 0.8372257351875305, -0.6369609236717224, 0.36452290415763855, 0.12580294907093048, -1.8183064460754395, -0.6529073715209961, 1.2306421995162964, 0.5572628378868103, -1.079432487487793, -1.1434651613235474, 0.9332480430603027, -0.7597323060035706, -1.3067303895950317, 0.548907995223999, -0.8307464122772217, -0.2603263258934021, -1.7212408781051636, -0.38226985931396484, -0.5695371627807617, -0.6191737651824951, 0.9943917989730835, 0.6727967858314514, 0.9845267534255981, 0.03867676481604576, 0.32137593626976013, -0.21672141551971436, -0.20333066582679749, 0.49123474955558777, 0.4129120707511902, 0.06077837198972702, 0.2580356299877167, -1.972003698348999, -0.9445486068725586, -0.11357923597097397, 0.6302244067192078, -1.2983219623565674, -0.2314191311597824, -0.49852579832077026, -0.17755994200706482, -1.720025897026062, 1.1688154935836792, 0.5659592747688293, -0.05867734178900719, 0.27455517649650574, 0.18723639845848083, 0.1539374589920044, -0.4071081280708313, -1.971582055091858, 2.2507717609405518, 1.5105046033859253, 0.6415244936943054, -0.39566805958747864, -2.00950288772583, 0.6954473257064819, 0.3506520688533783, -0.6425699591636658, 0.4869546890258789, 2.641517400741577, 0.013342141173779964, -0.7903038263320923, -0.7341143488883972, 0.5907970070838928, 0.3506145775318146, 1.4942233562469482, -0.8105072379112244, 1.7105724811553955, -1.561660647392273, 0.08029046654701233, 0.6611067652702332, 0.3919813632965088, 0.18844665586948395, -0.4895436763763428, -0.6097185611724854, -0.15770576894283295, 1.3570395708084106, -0.5210151672363281, -0.22555433213710785, 2.1085078716278076, 0.45096269249916077, -0.23573879897594452, -0.4609023928642273, 0.11230740696191788, -0.6753876805305481, -1.367400884628296, 1.1928832530975342, -1.0935708284378052, 0.11295414716005325, 0.22886979579925537, -0.8467085957527161, -0.31712329387664795, 1.4388606548309326, -0.049833882600069046, 2.302978038787842, 0.8380677103996277, 0.9394721388816833, 0.5546316504478455, 0.13387282192707062, -0.36702772974967957, -1.0949329137802124, -1.3199926614761353, 0.8078030347824097, 0.08369773626327515, 0.21630941331386566, 0.9396522641181946, -1.5835267305374146, 1.316041350364685, -1.7087602615356445, -0.013199192471802235, -0.4957945644855499, -1.4195979833602905, -1.0113420486450195, 0.9754958152770996, -0.00868055410683155, -0.4914392828941345, 0.4757145643234253, 0.051060356199741364, -0.4961658716201782, -0.02735089138150215, -0.9351438283920288, 0.6980396509170532, -0.8955549597740173, 1.3974800109863281, 1.5277142524719238, 0.45474329590797424, -0.7718181014060974, -0.9802088737487793, 0.9009638428688049, 1.7515993118286133, -1.8877495527267456, -1.8364592790603638, -1.0143365859985352, -0.911864697933197, -0.5827220678329468, -0.4499765634536743, -0.1803857386112213, -0.623485267162323, -0.0919589251279831, 1.6812264919281006, -1.0590178966522217, 1.2871954441070557, -1.1348800659179688, 0.1699906289577484, 0.01732383668422699, -0.32552221417427063, 0.3219756484031677, 0.20475457608699799, 0.30199599266052246, -0.8533220291137695, 1.7215656042099, -0.7268267273902893, 0.9213901162147522, 1.239872932434082, -0.31704282760620117, -0.9028030633926392, -0.4466407299041748, 0.7238711714744568, 0.41129425168037415, -1.2072902917861938, 0.6631190180778503, 0.6946938633918762, -0.9001129865646362, 0.39781641960144043, -0.8062043786048889, 0.7615485191345215, -1.837131142616272, -1.9618417024612427, -1.061330795288086, 0.24625787138938904, 0.2835371494293213, 2.9264538288116455, 0.10895027965307236, 1.1141005754470825, 0.7175306081771851, 0.7174993753433228, 0.2831907868385315, -1.273550271987915, 1.8583903312683105, -0.36013880372047424, -1.5715316534042358, 0.2904505133628845, 0.05851186439394951, 0.13323290646076202, 0.9075486660003662, -0.8307125568389893, -0.7011347413063049, -2.0891079902648926, -0.44956696033477783, 0.42115259170532227, -1.472580075263977, 1.2748898267745972, 0.5387108325958252, -2.0919301509857178, -0.4999469518661499, -0.012987440451979637, -0.05440247431397438, -0.168192520737648, 0.22463566064834595, 1.666225790977478, 0.15215352177619934, -0.15054385364055634, 1.052528738975525, 1.2656521797180176, -0.9699751138687134, 0.9714210033416748, 0.6879343390464783, 1.605691909790039, -0.28383439779281616, 0.06890059262514114, -0.5613632202148438, -0.37019312381744385, -0.6661803126335144, -1.1029750108718872, 0.032991211861371994, 1.1879826784133911, 0.4048197865486145, 0.33242085576057434, 0.15433788299560547, 0.9766177535057068, 0.6471353769302368, -1.1427030563354492, 1.9574487209320068, -1.7947486639022827, 0.89399254322052, 1.3413444757461548, -1.9783505201339722, -0.6586007475852966, -1.809116005897522, 0.45298781991004944, -2.316704511642456, 1.5330991744995117, 0.8062326312065125, -1.017707109451294, 0.0716574490070343, 0.7500154376029968, 0.43021178245544434, 0.7950800657272339, 1.6641894578933716, 0.2322596311569214, 0.4459613859653473, 0.8049592971801758, 1.3177565336227417, 0.1799570769071579, 0.07045985013246536, 0.479294091463089, 1.947738766670227, -0.9451225996017456, 0.5492151379585266, -1.6369683742523193, -0.4155884087085724, 0.40432700514793396, 0.7370054125785828, -0.7717494368553162, -0.9361306428909302, 1.2129795551300049, -0.3142275810241699, -0.9153962135314941, 0.510064423084259, 0.7497014999389648, -1.9036054611206055, 1.04707932472229, 0.6289797425270081, -2.204580545425415, 0.7680766582489014, -1.312379240989685, -0.2357175201177597, 0.46699830889701843, 0.06573749333620071, -0.021449336782097816, -0.10512642562389374, 1.5209568738937378, 0.02053757570683956, -1.3748897314071655, -0.026885442435741425, -1.709235429763794, -0.43077000975608826, 0.4960179328918457, -0.9095849394798279, 0.8542140126228333, -1.5639399290084839, -0.30163663625717163, -0.32647839188575745, -1.6021026372909546, 0.761106014251709, -0.23403297364711761, 0.0884050577878952, -1.2451879978179932, 0.8917390704154968, 2.51945161819458, 0.4278336465358734, 0.9731777310371399, 1.3806850910186768, -0.43426647782325745, -0.9291874170303345, -0.6940967440605164, -0.011121428571641445, 0.3163413405418396, -1.3929225206375122, -1.4109432697296143, -0.7822822332382202, 0.19373255968093872, 0.6810517907142639, 0.9603513479232788, 1.125663161277771, 0.22687622904777527, -1.097556471824646, 0.017933957278728485, -1.1192197799682617, 1.1858994960784912, -0.04549102857708931, -1.7795894145965576, 0.302304208278656, 0.9785152673721313, -1.3992239236831665, -2.1936604976654053, -0.551688551902771, -0.09034279733896255, 0.241624116897583, 0.07502410560846329, 1.1642260551452637, 1.43178129196167, 0.9256842732429504, 1.9891213178634644, -0.40337494015693665, 0.1045922040939331, -0.6724551320075989, 0.2533775269985199, 0.49875155091285706, -0.1846645623445511, -1.7035033702850342, 1.033244252204895, 0.4740753471851349, 1.6783229112625122, -1.735358715057373, 1.2021613121032715, -0.8320797681808472, -1.9972296953201294, -0.22147196531295776, -2.232710123062134, -0.011848759837448597, -0.5805435180664062, -0.13970571756362915, 1.010303020477295, 2.23622989654541, 0.8350769281387329, 1.502200722694397, 0.018609050661325455, 0.20984207093715668, -1.9064252376556396, 1.0081918239593506, -0.06120590493083, -0.637483537197113, 0.8554670214653015, 1.0337082147598267, -0.4113335907459259, -1.6113502979278564, 1.2890949249267578, 0.15028047561645508, 1.249312400817871, 0.281402051448822, 0.9929556250572205, -0.7264314889907837, 1.4309133291244507, 0.05041336640715599, 1.4904472827911377, 1.524185061454773, -0.7943282127380371, 0.25203806161880493, 0.7274873852729797, 0.11586490273475647, -1.1573047637939453, -0.8632749915122986, 0.8626213073730469, -0.1584896296262741, 0.11315829306840897, -0.05133504047989845, -0.8696801066398621, -1.1483502388000488, -1.948126196861267, 1.9425488710403442, -0.3620603084564209, 2.225135326385498, -1.4594942331314087, -2.3828046321868896, 1.071489930152893, 0.4640916883945465, 0.045263033360242844, 0.654127299785614, 1.554365873336792, -0.15464478731155396, -1.346746563911438, -1.0614831447601318, -0.6299917101860046, -0.2843051552772522, -0.9901527762413025, -0.9516140222549438, 0.1352963000535965, 1.301059365272522, 1.300608515739441, -0.299355685710907, 0.9806838631629944, -0.42023056745529175, 3.1015872955322266, -0.15425260365009308, -0.39844995737075806, 1.432844638824463, 0.19149234890937805, -1.5981216430664062, 0.29061904549598694, 1.061013102531433, -1.7879716157913208, 0.6545186638832092, -0.7457764744758606, -1.425261378288269, 0.09811289608478546, -0.36970990896224976, -0.1022547036409378, 1.4916508197784424, -1.4349743127822876, 1.0601238012313843, -0.2107033133506775, 0.732835590839386, -0.679751992225647, 1.6063265800476074, 0.029816433787345886, -0.15942031145095825, 0.22943761944770813, -0.4447593688964844, -0.46542900800704956, 2.3417699337005615, 2.3388454914093018, -0.20781686902046204, 0.7848660349845886, 0.2003801465034485, 0.4294668436050415, -1.1235897541046143, -1.425688624382019, 0.07988664507865906, 0.7276358604431152, 1.3230761289596558, -0.35265985131263733, -0.13521340489387512, 0.6153466701507568, 0.7531098127365112, -0.06310493499040604, -0.3125711679458618, -2.430446147918701, -1.3357207775115967, -0.5089478492736816, -0.4341937005519867, 0.09167273342609406, 0.9460211992263794, -1.234907627105713, -0.33576929569244385, -1.1229372024536133, 0.29279279708862305, 0.5745055079460144, -0.8085339069366455, -1.5801570415496826, -2.8058254718780518, 0.27795523405075073, -0.9495672583580017, -0.8166925311088562, 0.9735988974571228, -0.6662596464157104, -0.7818953990936279, -1.7131292819976807, -0.7895717620849609, -1.0121612548828125, -0.6059255003929138, -1.213117241859436, 0.007834630087018013, -0.37486732006073, 0.6189388036727905, 0.004346328321844339, 0.2264668047428131, 0.2815415561199188, 0.5888626575469971, -1.7359086275100708, 0.7159214019775391, 0.8795116543769836, 0.6758230924606323, 0.5718675255775452, 0.19655419886112213, -0.6223263740539551, -0.48076775670051575, -0.015226083807647228, -0.9113138914108276, -0.6334329843521118, 0.5801702737808228, -2.614650249481201, -0.027293307706713676, 0.26370447874069214, 0.01899775303900242, -1.1482458114624023, -1.3841146230697632, -1.036170244216919, 0.6753990650177002, -0.9117481112480164, -1.2625354528427124, -0.5355947017669678, 0.10673172026872635, -2.4964990615844727, -1.7438584566116333, -0.35901933908462524, -1.1494733095169067, 0.8360106348991394, 1.6148302555084229, 1.8192113637924194, -0.6106328964233398, 1.236201286315918, 0.4167012572288513, 1.7656216621398926, 1.1616816520690918, 1.2430225610733032, -0.09251624345779419, -0.5103650689125061, 1.6011055707931519, 0.8457103967666626, -1.1078096628189087, -1.0344913005828857, -0.11737945675849915, -0.94953852891922, 0.2643941640853882, -0.23332656919956207, -1.2285926342010498, -1.176988959312439, 0.9292769432067871, -0.5954235792160034, 1.1185393333435059, -0.2136256843805313, -1.1094996929168701, -0.2928980886936188, -0.19201260805130005, 0.571938693523407, -1.7441786527633667, 1.1362708806991577, -0.4593521058559418, 0.01681119203567505, 0.050885360687971115, -0.9220768213272095, -0.15796655416488647, 1.473439335823059, -0.38220423460006714, -0.8732162714004517, 0.22747118771076202, -0.8760210275650024, -2.101656436920166, 0.7124857306480408, 1.3292075395584106, -0.0708802193403244, 0.5800883173942566, -0.21307425200939178, -1.710530161857605, 0.7768727540969849, 0.4861806035041809, -0.5053932070732117, 0.8227820992469788, -1.616106629371643, -0.2820301651954651, 0.5756747722625732, -1.4663825035095215, -0.36470046639442444, 0.0028665054123848677, -0.5835832953453064, -0.5258747935295105, 0.6078851819038391, 0.13134260475635529, -1.1366835832595825, -0.22551442682743073, -1.1490535736083984, 0.3794921636581421, -0.04252717271447182, -0.8827417492866516, -0.07724785059690475, -0.2697155475616455, -0.5391042232513428, 2.7396769523620605, 0.6567059755325317, 1.3103303909301758, 0.7493577599525452, 0.04668391868472099, 0.46753257513046265, 1.8098437786102295, 0.21295247972011566, 0.6304096579551697, 0.045879025012254715, -1.207486629486084, 0.21968179941177368, 0.4821493923664093, 0.8152306079864502, -0.684145450592041, -0.39091596007347107, 2.107618808746338, -0.0990179106593132, 0.7039577960968018, -0.6369280815124512, -0.6946144700050354, 1.182843565940857, 0.1347237229347229, 1.6541699171066284, -0.5056531429290771, 0.9920597672462463, 0.308425635099411, 0.6858859658241272, -0.24174833297729492, 0.11217094212770462, 0.10854795575141907, 0.6708464622497559, -0.4809146225452423, -0.21315918862819672, 0.10412225872278214, 2.2105934619903564, 1.8348098993301392, 1.1040165424346924, -1.5647470951080322, -0.9785467386245728, 0.08813993632793427, -0.16261647641658783, -1.5190513134002686, -0.757316529750824, 1.022780179977417, 0.0788625031709671, -0.08151960372924805, -0.16732178628444672, -0.6686125993728638, 0.35094764828681946, 0.21223010122776031, -1.1310487985610962, 2.4008936882019043, 0.2147788107395172, -0.8467687964439392, -1.355734944343567, 1.691854476928711, -0.9740726351737976, -0.01957768201828003, -1.421363115310669, 1.11246657371521, 1.4398106336593628, -0.36886632442474365, -0.5584477782249451, 1.168135643005371, -1.1999317407608032, -1.7116831541061401, 0.8959149718284607, 1.5557944774627686, 1.2853585481643677, -0.1276836395263672, -0.3501059412956238, -0.6432890295982361, 0.8473437428474426, 1.5114948749542236, 0.9404686689376831, -0.031256724148988724, 1.4602992534637451, -0.16957740485668182, -0.8603202104568481, 0.5510148406028748, -1.180981159210205, -0.354020357131958, -1.3991761207580566, 0.2378532886505127, 0.960541844367981, 0.746678352355957, 0.6101351380348206, 0.10190674662590027, -0.5249617695808411, -1.022736668586731, 0.10811927169561386, -1.325108528137207, 0.890286922454834, -0.3274649381637573, -0.6348749995231628, -1.1919595003128052, -0.5159190893173218, -1.1080000400543213, 0.5636428594589233, -0.005000226199626923, 0.00929240882396698, 0.5454881191253662, -1.0161428451538086, 0.8058409094810486, 1.2854621410369873, 0.057644736021757126, 0.12311053276062012, 1.1829075813293457, -1.128965973854065, -0.08651957660913467, -0.3742310106754303, -0.3763462007045746, 1.551456332206726, -1.4163832664489746, -1.1309696435928345, -0.12628740072250366, -1.6668773889541626, -1.7026087045669556, -1.4243985414505005, -0.12096572667360306, 0.44488877058029175, 0.3775591552257538, -0.5042644739151001, 1.6973235607147217, 0.1922445446252823, 1.5287681818008423, -1.4254090785980225, -1.0559916496276855, 0.008727645501494408, -0.4707739055156708, 0.8540278077125549, 1.4579459428787231, -0.5689855217933655, 1.5519819259643555, 0.8621035814285278, 1.561550259590149, -1.801746129989624, 0.737630307674408, 1.6886496543884277, -0.1759723722934723, 0.4647246301174164, -0.2775830030441284, -0.9377334713935852, 0.01429846603423357, -1.5694295167922974, -1.825100302696228, 1.564687967300415, 0.3751099109649658, 1.6374884843826294, -1.174545168876648, 0.5684293508529663, -0.6567080616950989, -0.5405900478363037, -0.7174168229103088, 0.3027918040752411, 1.3723344802856445, -0.7826502919197083, -0.8486022353172302, -1.4569065570831299, -1.3716291189193726, -2.1131577491760254, -0.032630059868097305, -0.43987664580345154, -0.06261374801397324, 0.1387389749288559, 1.483188509941101, -2.025210380554199, 0.6846447587013245, 0.3686065971851349, 0.9853066802024841, 1.5043524503707886, 0.6777247786521912, 1.7177013158798218, 1.2637755870819092, -0.5882625579833984, -1.6106232404708862, 0.45397526025772095, -0.7614907622337341, -0.6894927620887756, 2.2061126232147217, 0.23520512878894806, -2.092170476913452, -0.3646441400051117, -0.37606680393218994, 1.2649489641189575, -1.5464287996292114, -0.8346366882324219, 1.2419567108154297, 1.8185349702835083, 0.4269085228443146, 0.4924618899822235, 1.1066317558288574, -1.604884386062622, 0.09371963888406754], [-0.8918489217758179, -1.7479678392410278, 0.6904534697532654, -0.8074032068252563, 0.035566676408052444, -0.24704957008361816, 1.045751690864563, -0.3531622886657715, -1.0215905904769897, 0.6404038667678833, -2.4779515266418457, 1.7459423542022705, -0.36428794264793396, 1.724749207496643, -1.613448977470398, 1.1520445346832275, 0.016023896634578705, -1.1309255361557007, 0.19663657248020172, -0.3947918713092804, 0.6542608737945557, 0.21655263006687164, -1.5623546838760376, -1.7611284255981445, -1.7819198369979858, 0.0006466752383857965, -0.684520423412323, -0.9696342945098877, 0.07483500242233276, 0.8736007213592529, -1.627640962600708, -0.8407701253890991, -1.1849242448806763, 0.32127219438552856, 1.9544328451156616, 0.7951434254646301, 0.167164608836174, 1.5635496377944946, -1.886902928352356, 1.4040535688400269, 0.13117174804210663, 1.228540062904358, -2.097975015640259, 1.9044839143753052, -0.3567977249622345, -0.15415059030056, -1.5217667818069458, -0.3620045483112335, -0.5443794131278992, -0.14313678443431854, -0.40894824266433716, 0.5824981927871704, -0.11225675791501999, 1.037157654762268, 1.1783123016357422, 0.26901987195014954, 1.1412930488586426, 0.7436091899871826, 1.227073073387146, 1.2011845111846924, -1.6208131313323975, 2.4201178550720215, -0.920106053352356, -0.27490028738975525, -2.581895589828491, -0.060978710651397705, -1.140113115310669, 2.611710548400879, 0.33913737535476685, 0.6929426789283752, -0.7814866900444031, 2.158186197280884, 1.1217176914215088, -0.5829968452453613, -0.7476791143417358, 0.6164073348045349, 1.7080495357513428, -1.2596105337142944, 0.04574327543377876, 0.7485931515693665, -1.0283188819885254, 1.8581933975219727, -0.43185093998908997, -1.8098608255386353, 0.7706935405731201, -0.4840882420539856, 0.40526995062828064, -0.5062628984451294, 0.11920295655727386, -1.694133996963501, -1.9987455606460571, -1.566888451576233, 0.3694521486759186, -0.6449926495552063, -0.001957518979907036, 0.07855898886919022, 0.14076955616474152, 0.13187243044376373, 0.9696189165115356, 0.9599258899688721, -0.8570902347564697, 0.4004967510700226, -0.4174104928970337, -0.03247740864753723, -0.2721259891986847, 1.3852778673171997, 1.4155703783035278, -0.3217840790748596, -0.9526281356811523, 0.03417917340993881, -0.7181522846221924, 0.07493410259485245, 0.4715394377708435, -0.8825073838233948, -0.8418334126472473, 1.6885169744491577, 0.03732186183333397, 0.4242969751358032, 0.7601494789123535, 0.9826980829238892, 0.4443729519844055, -0.34924179315567017, -0.5141058564186096, 1.153624415397644, -0.5194051861763, -0.6199309825897217, 1.263910174369812, 3.721287965774536, 0.9135929346084595, 0.09987011551856995, 0.029397573322057724, 1.0916866064071655, -0.4480220377445221, -0.07843980193138123, 0.5056338310241699, -1.1405110359191895, 0.43577924370765686, 0.9561859369277954, 1.078251838684082, -0.2683754563331604, -0.21372798085212708, 1.6531199216842651, 0.01870862953364849, -1.2969698905944824, 0.22025547921657562, -1.4836857318878174, 0.3962603211402893, -0.08409935981035233, 0.1252126544713974, -1.4106324911117554, -1.3931071758270264, -0.8730757236480713, 0.9574530720710754, -0.4324798583984375, -0.07992006093263626, -1.2926146984100342, -0.9126117825508118, 0.4931562542915344, 0.6118127107620239, 2.477546215057373, -1.5262327194213867, 1.0290195941925049, -1.2230063676834106, -0.6862455010414124, 1.1307183504104614, 0.5556883811950684, 1.4019194841384888, -0.09982848912477493, -0.7136512398719788, -0.6662569046020508, -0.5872251987457275, 1.3379888534545898, -0.46397995948791504, -0.32617276906967163, -0.9958604574203491, -0.1863183230161667, -2.2326905727386475, -0.03360849991440773, -1.1691772937774658, -1.0392013788223267, -0.48432618379592896, 0.10312744230031967, 0.205870121717453, -0.31675124168395996, -0.5861018300056458, -0.2374792844057083, 0.5070506930351257, -0.12853644788265228, -0.9015589952468872, -1.4549100399017334, -1.7425549030303955, 0.12409728020429611, 0.10150764882564545, 0.973552405834198, 0.47038355469703674, 1.4805078506469727, 1.4941279888153076, 0.028344308957457542, 1.203816533088684, 1.2699319124221802, 0.7803354859352112, -0.6268550753593445, -0.40016525983810425, 0.5210500359535217, 1.5294830799102783, -0.43564993143081665, -0.038637202233076096, -0.8982723355293274, 0.8006834387779236, -0.12777404487133026, 0.4201611578464508, -0.39178553223609924, -0.8915746808052063, 0.6563677787780762, -0.2108304798603058, 0.8087204098701477, 0.06838660687208176, 1.7630047798156738, -1.5319167375564575, -3.4020497798919678, 0.43410515785217285, 0.25774461030960083, -0.30864691734313965, -1.3523077964782715, -1.8651772737503052, -0.7906712293624878, -0.1351674199104309, -0.5936629176139832, 1.5086100101470947, -0.6005032658576965, 0.7058902382850647, -0.8142763376235962, -0.9648563861846924, 0.16357284784317017, 0.8763478398323059, 0.655251681804657, 0.6186913251876831, -1.0270441770553589, -1.2245278358459473, -0.20851387083530426, -1.0228866338729858, 0.5711883306503296, 1.0530500411987305, -0.6046113967895508, -0.4637530744075775, -0.2514641582965851, 0.37172603607177734, 0.6680201888084412, 0.44718092679977417, 0.6445898413658142, 0.2839936912059784, -0.32444700598716736, 0.19244781136512756, -1.7358149290084839, 0.15386316180229187, -0.27424219250679016, -0.5436712503433228, 0.9853981733322144, 1.2450124025344849, 0.35869836807250977, 0.861465573310852, -0.7306682467460632, -0.7719249725341797, 0.2863987386226654, -0.43681928515434265, -0.4556722939014435, 0.5055245757102966, 0.7126759886741638, 0.12864382565021515, 0.5901750326156616, 0.19249534606933594, 0.0034971144050359726, -0.882925271987915, 0.8926404714584351, 1.6954116821289062, -1.1218525171279907, 0.7259613275527954, -0.5808638334274292, 0.5750519633293152, -0.32940900325775146, -2.872925281524658, 1.8399015665054321, -0.18635602295398712, 0.2818506062030792, 0.8023450374603271, 0.8621276617050171, -0.6531816124916077, -0.16872748732566833, 0.7303506731987, 0.8913134932518005, -1.3362007141113281, 0.15716660022735596, -2.197460889816284, 0.29869502782821655, -2.4558682441711426, -0.43379995226860046, 0.3331783711910248, 0.8319261074066162, -0.024839293211698532, -0.18077006936073303, -0.042263127863407135, -0.2698618173599243, -0.3747268617153168, 1.558675765991211, 0.5873115658760071, 0.13867707550525665, 0.7995336055755615, 1.689295768737793, 0.18430747091770172, 0.5432718396186829, -1.426587462425232, 1.1436028480529785, -0.32262974977493286, 0.39237481355667114, -0.39133328199386597, -0.4702313244342804, 0.3033140003681183, -0.5531448125839233, 1.6332708597183228, -0.210356205701828, -1.6221377849578857, 0.8806426525115967, 0.23420266807079315, -1.0606956481933594, -1.1463854312896729, 1.3878213167190552, -2.2058968544006348, -0.09010418504476547, 1.0528079271316528, -0.35484108328819275, 0.2760690748691559, 0.3403114378452301, -0.5839813947677612, 1.7463358640670776, 0.24128809571266174, 0.2895644009113312, 0.6768072247505188, -1.4585200548171997, -0.7957198619842529, 0.014562893658876419, 0.764779269695282, 0.16093023121356964, -0.30429068207740784, 1.1476573944091797, -0.9085401892662048, -0.939505934715271, -0.7787662744522095, -0.934805154800415, 1.0675678253173828, 0.2531273066997528, -0.9882581233978271, -1.1069427728652954, -0.395418643951416, 1.095932126045227, -0.8353201150894165, -0.2002781480550766, 0.2852705717086792, 0.03421503305435181, -0.4019326865673065, 1.3461990356445312, -1.2819490432739258, 1.286927580833435, 0.3535148501396179, 0.3583909571170807, 1.5925779342651367, 0.658383846282959, 0.19773165881633759, -0.34907692670822144, -0.1387857347726822, 0.43337252736091614, 0.14021292328834534, 0.8513975143432617, 0.5255941152572632, -0.5782414674758911, 0.9557856917381287, -1.2673754692077637, 1.1334270238876343, -1.6816822290420532, 0.2829104959964752, 0.5501987934112549, 1.4409972429275513, 0.3292345404624939, 1.4050192832946777, 0.18729417026042938, 0.04817511513829231, -1.0665359497070312, -1.3355658054351807, 0.09260184317827225, 0.03933737799525261, -1.6149662733078003, -2.57289981842041, 0.5447630882263184, 1.4985852241516113, -0.8560498952865601, -0.9841760993003845, -0.6172763705253601, -0.3230999708175659, -0.8461956977844238, -0.6521385908126831, 0.5060706734657288, 0.43211829662323, -1.0199580192565918, -3.0939271450042725, 0.4074438214302063, 1.6250513792037964, 0.19929185509681702, 0.12049715965986252, 0.6073091626167297, 0.05387081950902939, 0.2447921633720398, 1.729551911354065, -1.5662072896957397, 0.7440062761306763, -0.19714614748954773, -0.43965867161750793, -0.12493937462568283, 0.7674672603607178, 1.0642796754837036, 0.5339320302009583, 0.29294896125793457, -0.8350483775138855, 0.21132101118564606, -2.3173282146453857, 0.26687338948249817, 1.8103363513946533, -0.5468249320983887, 0.05905115231871605, -0.10149896144866943, -1.116852045059204, -0.9654473662376404, 0.048776667565107346, -0.747336208820343, -0.09279587119817734, -1.4735450744628906, 0.4400328993797302, 0.35255885124206543, 1.2944391965866089, -0.5803954601287842, 0.7254313230514526, 0.12548522651195526, 1.0834167003631592, 1.0601353645324707, 0.7205865383148193, 0.2839190363883972, 1.3251328468322754, 1.263495922088623, -0.877778947353363, 1.0622951984405518, -0.6933455467224121, -0.201304093003273, -0.4911968410015106, -0.9693521857261658, 1.2229018211364746, 0.38665977120399475, -1.3902796506881714, -0.7960048317909241, -0.6509328484535217, 0.6775923371315002, 0.30716729164123535, -1.5966218709945679, 0.06157263368368149, -0.2008373737335205, 0.3475739657878876, -0.4742380976676941, 0.5101823806762695, 1.2499573230743408, 1.3586689233779907, 0.3083420693874359, 0.3994082510471344, -0.6395807266235352, -0.8985698819160461, -0.6886264681816101, 0.7864516377449036, -0.5864648818969727, -1.1690924167633057, 1.7570641040802002, -0.25913700461387634, 0.2499893307685852, 0.09427344799041748, 0.5378717184066772, -1.232654094696045, 0.3413338363170624, 0.11199773102998734, 0.4395676553249359, -0.3384103775024414, 0.2874178886413574, -0.37598487734794617, -0.8198809623718262, 0.2078544646501541, -0.2449665665626526, -0.4481378495693207, 1.602076768875122, 1.2449287176132202, 0.4085816442966461, 1.283786416053772, -0.6311284303665161, -0.5837172865867615, -0.24184954166412354, 0.06511704623699188, -0.16204676032066345, 1.9808779954910278, -0.45473915338516235, 0.07949542999267578, -1.9240598678588867, 0.020778300240635872, 1.3295636177062988, -0.5371893048286438, -0.6252143383026123, 0.693715512752533, -0.3126274049282074, -0.14628180861473083, 0.8247734308242798, 0.17010095715522766, -0.06068173795938492, -0.3206654489040375, -2.022810220718384, 0.1654479205608368, -0.03625291958451271, 0.4229072034358978, -0.41450151801109314, 1.2658023834228516, -1.3261066675186157, -0.25307828187942505, 0.723792552947998, 0.8448663949966431, 0.4239000678062439, -0.08958993852138519, 0.8444331884384155, -0.2845594584941864, 0.057752612978219986, -0.16950847208499908, 0.17770498991012573, -0.1786031723022461, 0.9410068988800049, -0.9229803085327148, 0.6304398775100708, 0.18058152496814728, 0.6988403797149658, -1.6466389894485474, -0.1401309221982956, 0.07066566497087479, 0.18769125640392303, -1.083344578742981, 1.2026660442352295, 0.0018481372389942408, -0.34730905294418335, 2.162306785583496, 0.06080801784992218, 1.545404314994812, -1.1310241222381592, 1.1823840141296387, -0.1851440668106079, 1.0077224969863892, -0.383789986371994, 0.7980212569236755, 0.4100957214832306, 1.1421598196029663, 1.9005982875823975, 1.3624062538146973, 0.9514869451522827, -1.3834997415542603, -0.12147766351699829, 1.652316689491272, 0.4247901439666748, 0.4195207953453064, 0.6358086466789246, -0.8177857995033264, -1.1928749084472656, -1.4490925073623657, -1.3657691478729248, -0.42558202147483826, -1.0324879884719849, 1.1791496276855469, -0.4635017216205597, -1.5072531700134277, 0.1406630277633667, 0.299052357673645, 0.27318140864372253, -0.5957044363021851, 0.463090181350708, -0.4771757125854492, -0.8889957666397095, 0.5271950960159302, -1.394684910774231, -0.7816488742828369, -1.3902312517166138, 1.433603048324585, -1.4332613945007324, -0.5132103562355042, -0.3391793370246887, -0.6432024836540222, 1.2996972799301147, -0.1412862092256546, -0.4274737238883972, 0.620255172252655, 0.17081135511398315, -0.5379627346992493, -0.7317201495170593, 0.12919799983501434, -0.45593854784965515, 0.8305955529212952, -0.8764415979385376, -0.2870636284351349, 0.674936056137085, 1.1445566415786743, 0.8643999695777893, -0.2720523476600647, -0.5665790438652039, -0.9038700461387634, -0.270230233669281, 0.9777291417121887, 1.4088897705078125, 0.13340230286121368, 0.575611412525177, -2.2366538047790527, 0.07539094239473343, 0.8667615652084351, 1.042891025543213, -1.6180444955825806, 1.0389598608016968, 0.3248635232448578, -0.359260231256485, 0.9124355316162109, -0.333549827337265, 0.09146280586719513, -1.332624077796936, 0.36463722586631775, 1.129353642463684, 2.2024993896484375, 1.0853523015975952, 1.3753043413162231, -0.8185768127441406, 0.04182034358382225, -1.491119384765625, -1.2007360458374023, -1.0871142148971558, 0.8187610507011414, 0.6578975915908813, 0.04148206114768982, 1.1971259117126465, 0.41504502296447754, 1.0364910364151, -0.05653451383113861, -0.341886967420578, -1.5991829633712769, -0.03970671072602272, 1.865994930267334, -0.5296855568885803, 0.8766899704933167, -1.330135464668274, -1.442333459854126, 0.37954258918762207, 0.0622788667678833, -0.35320693254470825, -0.6104472279548645, 0.9859106540679932, -0.8451019525527954, -0.07922904193401337, 0.40321841835975647, 0.9083623290061951, -1.3441270589828491, 0.3250304162502289, 1.2904189825057983, -0.16529175639152527, -0.8836559057235718, 0.44182729721069336, 0.6291174292564392, -1.8189384937286377, 0.6405609846115112, -0.1638151854276657, -0.8953303694725037, -0.18230809271335602, 0.7915118932723999, 0.8557322025299072, -2.0406622886657715, 0.524020254611969, 1.6973017454147339, -2.2334890365600586, -0.38953033089637756, -0.9806805849075317, -0.9561749696731567, 0.6234989762306213, 1.2053532600402832, -0.45715466141700745, 1.1796382665634155, 1.2292447090148926, 0.13730667531490326, -0.7155399322509766, -0.4882704019546509, 0.4769294857978821, -0.02556215599179268, 1.1931509971618652, 0.13341489434242249, -0.43026477098464966, -0.1452544778585434, -0.1533299684524536, -0.28563761711120605, -0.8063987493515015, -0.17517231404781342, 0.6194689273834229, -0.4738673269748688, 1.4208271503448486, 1.0478482246398926, 0.2775409519672394, -0.2158365249633789, -2.2692296504974365, -1.1711039543151855, 0.16665223240852356, -0.6317991018295288, 0.8881810903549194, -0.609049379825592, -0.21068315207958221, 0.1982761025428772, -0.8622934818267822, 0.8872328996658325, 0.942001461982727, -1.7020827531814575, -0.07353343069553375, 0.9810510277748108, -0.5846692323684692, 2.462355852127075, -0.10773243010044098, -0.18248702585697174, 0.3787693679332733, 0.8618859648704529, 0.7232474088668823, 0.7861822247505188, 1.1125165224075317, 0.9185059666633606, 0.8227400183677673, -0.6283262372016907, 0.9296694397926331, -0.10029856860637665, 0.4640067219734192, -0.8910542726516724, 0.5785555839538574, 1.6035348176956177, -0.8943240642547607, -0.026903124526143074, -0.08186306059360504, 0.1782175451517105, -0.07368331402540207, 0.4179767370223999, -0.4079328179359436, -0.13331840932369232, -0.06963848322629929, -0.4287199079990387, 1.4817482233047485, -0.09162963181734085, 1.6673036813735962, -0.49292781949043274, -0.4835663139820099, 0.10443923622369766, -0.9947407841682434, 0.20638886094093323, -1.0199394226074219, 0.904849112033844, 0.2803982198238373, 1.5984042882919312, 1.7326475381851196, 0.32778167724609375, -0.5242852568626404, -0.46568751335144043, -1.7628906965255737, -0.3984173834323883, -0.6173065900802612, -1.0392922163009644, -0.5683039426803589, 0.2923617959022522, -0.31134843826293945, 1.4767829179763794, 0.7162870168685913, 0.8816218376159668, -0.1445043683052063, 0.4042634963989258, 0.27640271186828613, -0.8699197173118591, 0.3963100016117096, -0.8631802797317505, 1.0641002655029297, -0.4369213283061981, -0.9848789572715759, 0.007663541007786989, -0.354223370552063, -0.4900255799293518, -0.48634225130081177, -0.7113985419273376, -0.5192379951477051, -0.3805544376373291, 0.11670384556055069, 0.9512255787849426, -0.44284558296203613, 0.7943226099014282, -0.740999162197113, -1.7605223655700684, -0.927734375, -0.0976560115814209, 0.23894131183624268, -0.7029807567596436, 0.16816285252571106, -0.013899462297558784, -0.027376500889658928, 0.3681566119194031, -0.7716201543807983, 1.1314921379089355, -0.5649924278259277, 1.1965785026550293, -0.7868322730064392, 0.4574028551578522, -0.6226862668991089, -0.6031277179718018, -0.3909834623336792, -0.09446073323488235, 0.9879918098449707, 1.738424301147461, 0.39276498556137085, -1.4455479383468628, 1.3544442653656006, 0.39067521691322327, -0.9671979546546936, -0.640052318572998, -1.2825393676757812, -0.8754605650901794, 1.577234148979187, 0.4942852854728699, 0.5894880294799805, -0.05433202534914017, -0.8689813017845154, -0.0467468723654747, 0.7429835796356201, 0.7842342853546143, 0.10469964146614075, 0.6721544861793518, 0.5648913979530334, -0.9722262620925903, 0.6764566898345947, 1.0953233242034912, 1.5212080478668213, -0.6561099290847778, -0.9233285784721375, -0.46567249298095703, 0.7565575242042542, 0.20955468714237213, -2.103745698928833, 1.6604644060134888, 0.12956567108631134, 1.3135591745376587, -0.30433371663093567, -0.05244928598403931, -0.08941298723220825, -0.4019547998905182, 0.3289603590965271, 1.0489554405212402, -1.2384330034255981, -2.229907989501953, 0.9280917048454285, 1.4990451335906982, -0.04724323749542236, 0.7091944217681885, 1.006310224533081, -1.3551926612854004, 1.3963792324066162, 0.5498523712158203, -0.49623844027519226, 1.640852928161621, 0.15663956105709076, 1.9554789066314697, -1.3618018627166748, 1.294377088546753, -1.246781349182129, 0.8317095041275024, -0.33826133608818054, -0.07291225343942642, 1.0711767673492432, -0.7371188998222351, 0.1578192412853241, -0.2709277272224426, -0.6866440773010254, 1.1821339130401611, -1.445719838142395, 0.47216862440109253, -0.9929966926574707, -0.7797988653182983, 1.3923473358154297, 0.17980855703353882, -0.16059930622577667, -0.7599308490753174, -1.66410231590271, -1.4219231605529785, 1.1394461393356323, 0.8855929970741272, 0.5514320731163025, 2.1752915382385254, 2.3662614822387695, 0.9251753687858582, -0.23616036772727966, 0.27684614062309265, 0.03816402703523636, 1.2257680892944336, -0.36370962858200073, 0.705960750579834, -0.7147402763366699, 1.5823781490325928, 0.3910275101661682, 1.1658130884170532, -1.1071372032165527, -1.2150322198867798, -1.1399375200271606, 1.0629793405532837, -1.0089575052261353, 0.3318953514099121, -0.45211324095726013, 0.7389926910400391, -0.3264977037906647, -0.10933580249547958, 0.42658835649490356, -0.8269767761230469, 0.5097575783729553, -0.26520878076553345, -0.5046988129615784, -1.1212047338485718, 0.016586827114224434, 0.20191743969917297, 0.7538340091705322, -1.5209022760391235, -0.19300420582294464, -0.5384649634361267, -0.19502125680446625, 1.6881552934646606, 1.0351362228393555, -0.4903402626514435, -0.06308054178953171, 1.3566455841064453, 0.922081708908081, 0.5003710985183716, -1.0546908378601074, 1.023182988166809, -0.3678494393825531, -0.08676169067621231, 1.475498914718628, -0.8133392930030823, -1.2800188064575195, -0.07029254734516144, 0.2906993627548218, -1.200275182723999, 0.009269933216273785, -0.6092997789382935, -1.2088788747787476, -1.273789882659912, 0.5514556765556335, 0.285008043050766, 0.6770542860031128, 1.0610977411270142, 0.734066367149353, 0.011273184791207314, 0.09671328961849213, 0.028042878955602646, -0.8255375027656555, -0.29986417293548584, 0.34760263562202454, 0.25124239921569824, -2.194551944732666, -0.2743080258369446, 0.3334839940071106, -0.4740884304046631, -1.4823188781738281, 1.3828613758087158, 0.3158198595046997, 1.5291754007339478, -2.1333119869232178, -0.6712738275527954, 0.17086847126483917, 0.5389084815979004, 0.16299887001514435, -0.7475957274436951, 0.15591156482696533, -0.274417906999588, -0.2858639061450958, 1.046213984489441, 0.24029317498207092, -0.54001784324646, -0.3942132294178009, -0.12696237862110138, 2.340365171432495, -0.0341586172580719, 0.71177738904953, -1.2613686323165894, -0.25843366980552673, 0.7626017332077026, 1.1216466426849365, -0.06408796459436417, 0.6254992485046387, 1.030341625213623, 0.2908194065093994, 1.7312928438186646, 0.08750315755605698, -0.20482008159160614, -0.2276057004928589, 0.1985160857439041, 0.11203841865062714, 0.19674861431121826, -1.8808097839355469, -0.18002909421920776, 0.1783861219882965, -0.4565142095088959, 0.1547003537416458, 0.7945049405097961, -1.046047568321228, 1.4530494213104248, -0.06082185357809067, -1.284525752067566, -0.29166606068611145, 1.4841808080673218, -0.010115772485733032, 0.09554773569107056, 1.4890987873077393, -0.38104304671287537, 1.6980336904525757, 0.32190772891044617, -1.1029456853866577, 0.6586334109306335, -1.0746982097625732, -1.1144195795059204, 1.1222560405731201], [-2.1342544555664062, -2.7308576107025146, -0.564864456653595, 0.3668697774410248, 0.6190884709358215, 0.8874855637550354, -1.051318883895874, -0.2724206745624542, -0.9156952500343323, 0.8576564788818359, -0.5565953254699707, 1.8237565755844116, 0.31562259793281555, 0.15718461573123932, -0.5237511992454529, 0.7814367413520813, 0.6282962560653687, -0.7441405057907104, -0.6204590201377869, 0.12348689138889313, 0.9204590320587158, 1.2998368740081787, -0.850870668888092, -1.3468273878097534, -0.5471323132514954, -2.3876376152038574, -1.3134859800338745, -0.7969567179679871, 0.1628260463476181, -1.071973204612732, -0.19897232949733734, 1.966442584991455, 0.37962061166763306, 0.6289169192314148, 0.7291679978370667, -0.8752807378768921, 1.0803431272506714, 0.17970944941043854, -1.0662180185317993, 0.9882209300994873, 1.897200107574463, -1.3088271617889404, 1.0902137756347656, 0.5058897733688354, -0.7167070508003235, 2.2174465656280518, 0.49124664068222046, -1.6823898553848267, -0.47144851088523865, 1.9221277236938477, -0.49289095401763916, -0.4785611033439636, 0.3575434386730194, 1.29470694065094, 0.03033400885760784, 0.18458271026611328, 0.15242131054401398, 0.8883681297302246, -1.4856317043304443, 0.699949324131012, -2.7898054122924805, -0.7656466364860535, 0.16678327322006226, -0.24020829796791077, -1.5943152904510498, -2.2390763759613037, 1.9289377927780151, -0.30723837018013, -1.3605377674102783, -1.3984572887420654, -1.7770841121673584, -0.08703997731208801, -0.052004698663949966, 0.8462682366371155, -0.5402436256408691, -1.139805793762207, -1.0243552923202515, 0.20815733075141907, 0.2740018665790558, -0.9335557818412781, -1.5390019416809082, -0.0003873267851304263, 0.6279342770576477, 0.492900013923645, 1.7147334814071655, -0.5560951232910156, 1.0444713830947876, 0.670470118522644, -0.6299638748168945, -1.0677683353424072, 0.6706812977790833, -1.3482249975204468, 1.2464168071746826, 0.2682253420352936, -0.007490728981792927, -0.15199770033359528, 0.4637274742126465, -1.4498600959777832, 1.3230501413345337, 0.4746084213256836, -0.9815266132354736, 0.08979100733995438, 0.19480949640274048, 0.7200440764427185, 0.4407862722873688, -0.42831069231033325, -0.36675629019737244, 0.0282946415245533, 0.3107437491416931, 0.7788683176040649, -1.0912998914718628, -2.3837778568267822, 1.6828570365905762, -0.022649485617876053, 0.6553272008895874, 0.49431371688842773, -0.6344796419143677, 1.0136083364486694, -0.19364888966083527, -0.2476641833782196, -1.2397181987762451, 0.046624377369880676, -0.3901461958885193, 0.41524428129196167, -0.4675116539001465, -0.7628806233406067, 0.3540702164173126, -0.046189554035663605, -0.5949087738990784, -0.0916706919670105, 0.15926308929920197, 0.29565346240997314, 0.78242427110672, 0.6079332232475281, 0.8728637099266052, -0.1604740470647812, -0.02083810605108738, -1.4875679016113281, 1.9523037672042847, 1.3072620630264282, -1.632448673248291, 0.3905300796031952, 1.8896983861923218, 0.04219667613506317, -0.20263352990150452, -0.7395771741867065, -0.25466033816337585, 0.25136053562164307, 0.7890344262123108, -1.2932244539260864, 0.14490576088428497, -0.9696422815322876, -0.8503227233886719, 0.0007183970883488655, 0.6741563677787781, -0.08510393649339676, -0.2570714056491852, 1.0794272422790527, 0.6618611812591553, -0.6299911737442017, -0.7658370733261108, -0.37676772475242615, 0.7816233038902283, 0.28444454073905945, 1.4721218347549438, 1.4567008018493652, 1.1130428314208984, -1.9995677471160889, 0.7891457080841064, 0.24995896220207214, 0.2646293342113495, -0.6771166920661926, -1.3770700693130493, 0.07944376021623611, 1.056436538696289, 1.316162347793579, -1.5306236743927002, 0.6288670301437378, 0.3414643406867981, 0.30279502272605896, -0.5929821133613586, -0.3293420970439911, 1.3412625789642334, 0.6444884538650513, -0.3042261004447937, -0.9018687009811401, 1.1587164402008057, -2.3137261867523193, -0.9673269391059875, 0.15863746404647827, -1.8310909271240234, -0.20968765020370483, -0.448594331741333, -2.3340275287628174, -0.8250415921211243, 0.22476017475128174, 0.38346153497695923, 0.3730255365371704, 1.1272659301757812, 1.4194227457046509, -1.2968900203704834, 0.031937893480062485, 1.163203477859497, -1.1300538778305054, 0.7907576560974121, -0.23035715520381927, 0.22310848534107208, -1.9508085250854492, -0.28861960768699646, -0.8665868639945984, -0.702214241027832, -0.35461723804473877, 0.1966446489095688, 0.28750288486480713, -0.5144266486167908, -0.15229886770248413, -1.04118812084198, 0.30255937576293945, 0.7969825267791748, 1.215378761291504, -0.796129047870636, -0.5872502326965332, 0.08661001920700073, -0.26996466517448425, -0.796299397945404, -0.9758254289627075, -0.8772692084312439, 1.2824496030807495, -0.9260987639427185, -0.026340315118432045, -1.5859273672103882, -0.061027392745018005, -0.9858766198158264, 1.3216044902801514, 0.5451068878173828, 1.1899621486663818, -0.3649660050868988, 1.1162091493606567, 0.5317865610122681, -0.6702134013175964, 0.9123357534408569, 0.46951091289520264, 0.0909748449921608, -1.334709644317627, -0.347503125667572, -0.5256692171096802, 0.8791692852973938, -1.7196757793426514, 0.04862256720662117, 0.8684938549995422, -0.043787259608507156, 0.33625492453575134, -1.1658735275268555, -0.5953575372695923, -0.562809407711029, 0.0820833072066307, 1.425196647644043, 0.8814471960067749, -1.0402028560638428, -1.2126423120498657, 0.5369644165039062, -1.0915303230285645, -0.7373409867286682, -0.3244796693325043, 0.08750981092453003, 1.0998505353927612, -1.6369309425354004, -1.6866143941879272, -0.3489202857017517, -1.5115000009536743, -0.9595934748649597, -0.8568480610847473, -1.9185521602630615, -0.37687352299690247, 0.997065544128418, -0.7687845230102539, 0.22455884516239166, -1.2064416408538818, -1.4956508874893188, -0.2708331346511841, 1.0078779458999634, 0.6862438321113586, 1.5428390502929688, 0.4097185432910919, -1.301443099975586, -0.06493046134710312, 0.3192691206932068, -0.6924337148666382, 0.9094482064247131, 1.2346956729888916, 1.1718072891235352, -0.2724115550518036, 0.1886572688817978, -1.1847182512283325, -0.21442382037639618, -0.396955281496048, -0.7943069338798523, -1.7979917526245117, 1.475480556488037, -0.2845143973827362, -0.34186944365501404, 0.03923794627189636, 0.7695948481559753, 0.10031791031360626, -1.7403709888458252, 1.3187857866287231, -0.3405061662197113, -0.37756094336509705, -2.0188827514648438, -0.36688199639320374, 0.6249246597290039, 0.7836710810661316, 0.8492675423622131, -0.6790764927864075, 0.7299527525901794, 0.566632091999054, -0.9948994517326355, -0.25932303071022034, -1.5904440879821777, -0.673362135887146, -1.2975860834121704, 1.2088853120803833, 0.7187667489051819, 0.010685657151043415, 0.8531307578086853, -1.8645561933517456, -0.7994522452354431, -0.4437102973461151, 0.38947027921676636, 0.9653179049491882, 1.699110507965088, -2.145960807800293, -0.1277739405632019, -0.1608482301235199, -0.3047332465648651, 0.1624421775341034, 0.5067353844642639, -0.752414345741272, 0.8576480746269226, 1.8088375329971313, 1.1408060789108276, 1.0923596620559692, -1.2101449966430664, -1.9896128177642822, 0.4101196527481079, 0.644267201423645, -0.32327887415885925, -0.6194207668304443, 0.28196805715560913, -0.3512166738510132, -0.9153035879135132, -0.5645349621772766, 0.059356819838285446, -0.5739870667457581, -2.091283082962036, -0.4016835391521454, -0.9509498476982117, 0.6820561289787292, -0.7844487428665161, -0.7656805515289307, -0.1717054694890976, 0.7875219583511353, -0.5164839625358582, -0.5519126057624817, -0.8262785077095032, -0.2306850403547287, 0.5916176438331604, -0.08378813415765762, 0.40756475925445557, 0.06670131534337997, 1.0653650760650635, -0.054333023726940155, 0.4934954345226288, -1.1919803619384766, -1.1749340295791626, 1.6020100116729736, -1.2016361951828003, 1.746028184890747, -0.47221246361732483, -0.6182628273963928, -0.4852142632007599, 0.7515124678611755, 0.7334321737289429, 1.7004889249801636, -1.0529321432113647, 0.8581365942955017, -1.5800014734268188, 0.8001813888549805, -0.12547120451927185, 1.3891046047210693, -1.3204501867294312, 0.8137142062187195, -1.5917457342147827, 0.8720213770866394, -0.2478584498167038, -0.08822766691446304, -1.189985752105713, -0.23293326795101166, -0.37180855870246887, -0.609369158744812, 0.048142459243535995, -0.5043959021568298, -0.38585948944091797, 0.8730524778366089, 0.5188018083572388, 0.32527869939804077, -0.21622060239315033, -0.23240765929222107, 0.7352075576782227, 0.8062260746955872, 1.268618106842041, -0.665524423122406, -0.6247241497039795, -1.6398544311523438, -0.13589692115783691, -0.4655376374721527, 0.6164446473121643, 0.5019180178642273, -1.390472650527954, 0.4302065372467041, 0.7721276879310608, 0.46890488266944885, 0.7301878333091736, 1.5707581043243408, 0.17789950966835022, 0.025514069944620132, 2.4376630783081055, 2.39970326423645, 0.3501507341861725, 1.4427694082260132, -1.1384589672088623, 0.9577604532241821, -0.15960392355918884, 0.5494277477264404, -0.08197280764579773, 0.14328911900520325, -0.20622292160987854, -1.3004878759384155, 1.4410218000411987, 0.3985724151134491, 0.777117133140564, -0.6316201090812683, -0.10048504173755646, 0.36046501994132996, -0.6590991020202637, 0.5868045091629028, 0.5571763515472412, -0.3075137138366699, -0.378523051738739, 1.0271062850952148, -0.6044769883155823, -0.722578227519989, -0.36160728335380554, 1.1214344501495361, -1.6638057231903076, -0.7025109529495239, -0.7809265851974487, -0.17509688436985016, -0.4155499339103699, 1.5870596170425415, -1.6019614934921265, -1.7537331581115723, -0.9065271615982056, 0.5269508361816406, -0.1952560395002365, -1.4624598026275635, 0.21182557940483093, -0.18695521354675293, -0.1886088252067566, -0.4407002925872803, 0.2335309386253357, 1.975957989692688, 0.18144135177135468, -0.5871032476425171, -0.0023707938380539417, 1.6893234252929688, -1.5795280933380127, 0.016025034710764885, 0.22603723406791687, 0.7384048700332642, 0.7329034209251404, -1.0563750267028809, -1.37849760055542, 0.6171078085899353, 1.99237859249115, 2.114079475402832, -0.06472266465425491, -1.6943739652633667, -2.2062740325927734, -0.3459211587905884, -0.9414196610450745, -0.8635916113853455, 1.0842243432998657, 1.3347358703613281, -0.30691999197006226, 1.0072822570800781, 1.0037940740585327, -0.9252093434333801, -1.3744585514068604, -0.08944553136825562, 0.369880735874176, 0.45252037048339844, 0.04836086928844452, 1.06263267993927, -0.5399947762489319, -1.035233497619629, 0.06470636278390884, -2.489255666732788, 0.09509363025426865, 0.7888876795768738, 1.4494397640228271, -0.9261126518249512, 0.24138396978378296, -1.9572665691375732, 0.9703831672668457, -2.575056314468384, 0.33684054017066956, 0.9971047639846802, 0.41925182938575745, 0.9329227805137634, -0.17515943944454193, -0.17824673652648926, 1.6467466354370117, -1.1563471555709839, -0.9505044221878052, 1.068088412284851, -1.1722608804702759, -0.4149606227874756, -0.42415767908096313, -0.5955097675323486, -0.014855519868433475, -0.4112876355648041, 0.8954734206199646, -1.8309942483901978, 0.5391179323196411, -1.2620985507965088, 0.4630391299724579, 0.17509257793426514, -0.07443693280220032, -0.4706970453262329, -0.6842127442359924, 1.4653187990188599, 0.8167795538902283, -0.404670774936676, -0.14482319355010986, 0.029364878311753273, 0.3793487846851349, -0.7062003016471863, -0.8535369038581848, -1.5288639068603516, -0.011930023320019245, 0.016352500766515732, 0.6342480182647705, -1.1335843801498413, 0.404346764087677, 0.39721405506134033, 0.014507959596812725, 0.8397290706634521, -1.4383467435836792, -1.3128293752670288, 1.0247912406921387, 0.0951562374830246, -0.01772501692175865, -0.08223522454500198, 0.32108139991760254, -0.6887156367301941, 1.439851999282837, -1.731454849243164, 1.8975551128387451, 0.5497567057609558, -0.3170897662639618, -2.292447090148926, 0.09821884334087372, 1.240774154663086, 0.20411331951618195, 1.0557456016540527, 0.36464443802833557, 1.1048444509506226, -2.3067049980163574, -0.5520429015159607, -0.5832359194755554, 1.2316014766693115, 0.8225154876708984, 1.7669572830200195, -1.5241836309432983, 0.4859725832939148, -0.3002948760986328, -0.5550211071968079, 1.1665164232254028, -0.7781540751457214, 1.6380548477172852, 0.9937951564788818, 0.6035680770874023, 0.16204838454723358, -0.13738864660263062, 0.4438842236995697, -0.8953453302383423, 1.337116003036499, 1.6212499141693115, 1.2560187578201294, 0.5218974947929382, -0.42974525690078735, -0.22452537715435028, -0.3653305768966675, 1.3236652612686157, 0.125826895236969, 0.7880746126174927, 0.2816235423088074, 0.13831907510757446, -0.7406300902366638, 0.0772431269288063, -0.840906023979187, -0.5838133096694946, -0.1476900428533554, -0.6825708150863647, -0.18655802309513092, 1.1686500310897827, 1.140833854675293, 1.7101961374282837, 0.26485204696655273, 1.002602219581604, 1.3258541822433472, -0.8013188242912292, -0.4527914226055145, 1.105342149734497, -0.1251666098833084, 0.25279542803764343, 0.32112476229667664, 0.9621744155883789, 0.4086560010910034, 0.6669047474861145, -0.22101879119873047, -0.2380104660987854, -0.6994643807411194, 1.825129747390747, 1.205470323562622, 1.4912211894989014, 0.9860291481018066, -0.4137318730354309, 0.8450992107391357, -0.9494007229804993, -0.5156668424606323, -0.18696115911006927, -0.77034991979599, -0.03078349493443966, 0.1625877320766449, 0.07119428366422653, -0.4239111840724945, -1.0769414901733398, -0.7187948822975159, 1.942085862159729, -1.4446440935134888, -0.4575878977775574, -0.06440342217683792, 0.30773571133613586, -1.0485832691192627, -0.1036437451839447, 0.6315034627914429, -1.6076375246047974, -0.3491421937942505, -0.09454739838838577, 0.24467866122722626, -0.46780720353126526, 0.6109928488731384, -0.7401989698410034, 0.3084556460380554, -2.2478342056274414, 0.11599480360746384, -0.9105401039123535, -0.8429823517799377, 1.4569368362426758, -0.5175873637199402, -0.4931114912033081, 0.8999336361885071, 0.012469896115362644, -1.5187273025512695, 0.2998983561992645, 2.69193696975708, -1.875400185585022, -0.042658884078264236, -0.36296725273132324, -0.04407152161002159, 1.7895760536193848, 0.5975514650344849, -0.26183241605758667, 0.323957234621048, -0.5824332237243652, -0.5578922033309937, -0.17124298214912415, -0.1487901508808136, 0.8408348560333252, 0.5350869297981262, 0.906875729560852, -0.5908180475234985, 0.5091652870178223, -1.3425971269607544, 0.2062119096517563, -0.14328818023204803, -0.08467837423086166, 2.163111925125122, -1.968373417854309, 0.4547741413116455, 1.1696457862854004, 0.2609414756298065, 0.6831832528114319, 1.4511958360671997, 0.9526852965354919, 0.11170396208763123, -0.0640060305595398, 0.8365868330001831, -0.26894789934158325, -2.184370517730713, 0.4540514051914215, -1.184447169303894, 0.1642472743988037, -0.2845260500907898, -1.656445026397705, 1.7027769088745117, -0.011254615150392056, 0.4939691722393036, 0.12675432860851288, 0.0930691510438919, -1.9502334594726562, 0.010195505805313587, -1.1069743633270264, 0.5682185292243958, 0.15848317742347717, 0.3309462368488312, -0.17864926159381866, 0.20422086119651794, -0.6276026964187622, -1.2631765604019165, 1.317591905593872, 0.3661138415336609, -1.0010994672775269, -1.602638602256775, 0.9762418270111084, -1.8235294818878174, -0.6567826867103577, 1.4383859634399414, 1.1547271013259888, 0.29104307293891907, 0.8998252153396606, -0.11355096101760864, -1.3395936489105225, 0.2584347128868103, 0.08208674937486649, -0.47871577739715576, 1.2462446689605713, 0.1254766434431076, -0.04664158448576927, -0.4574725329875946, 1.1062504053115845, -1.0660747289657593, -0.17947787046432495, -0.07965277880430222, 0.09939935058355331, 1.2939221858978271, 1.4881402254104614, -1.8220655918121338, -0.6018260717391968, 1.1840870380401611, 0.9885216951370239, -0.16088232398033142, 0.1711871474981308, 0.49004462361335754, -1.0742549896240234, -0.7296094298362732, -1.8287203311920166, -0.6683493256568909, -0.3476124107837677, -0.825217068195343, -0.9917783141136169, -1.7151708602905273, -0.15570051968097687, -0.8601874709129333, -0.7034367322921753, -0.14474456012248993, 1.2666620016098022, -2.063397169113159, 0.9975886940956116, -0.6064100861549377, 0.23643997311592102, 0.5718347430229187, 0.8995634913444519, -0.14784830808639526, 2.181886672973633, -0.8658621311187744, 0.306261271238327, -0.22593459486961365, 0.07373030483722687, -0.23135118186473846, 0.08469989895820618, -1.2860242128372192, 1.2364410161972046, -1.6968142986297607, -0.500575840473175, 0.0625036358833313, 0.7058746814727783, -0.769076943397522, -0.05843503773212433, 0.16551242768764496, 0.8572924137115479, 0.6228686571121216, 0.058740951120853424, -0.6557667255401611, 1.4190878868103027, -0.038723383098840714, -1.1001049280166626, -0.05497131496667862, 0.5803276300430298, 0.7341758608818054, -1.5214036703109741, -1.368353009223938, -0.24580030143260956, -0.4216511845588684, 0.39548712968826294, 0.3357565104961395, -0.003480791812762618, 0.4250586926937103, -1.3884228467941284, 0.6900070309638977, 0.8002864718437195, 0.221768319606781, 0.8621352314949036, 0.2622729241847992, -0.947982132434845, 1.2077744007110596, -0.5325453877449036, -2.3539791107177734, 0.3260208070278168, 0.7400924563407898, -2.509434700012207, 0.8672565221786499, 0.7585556507110596, -1.8312731981277466, -0.7487665414810181, 0.1182534322142601, -0.550721287727356, -0.8185911178588867, 0.136480912566185, 1.0680075883865356, -0.28913700580596924, 1.0083510875701904, 0.749474048614502, -1.586427092552185, 1.1470874547958374, -0.5052675604820251, -1.2429252862930298, 0.08820508420467377, 0.0005131152574904263, 0.1372867077589035, -0.5416136384010315, -0.9090284109115601, -0.23935869336128235, -0.7961181998252869, 0.2746695578098297, 0.09480299055576324, 0.1019962802529335, 2.351045846939087, -0.7622764110565186, -0.39340469241142273, 1.0568774938583374, 0.6034795641899109, -0.027139632031321526, 0.8566085696220398, 2.336407423019409, 0.912316083908081, -0.5470400452613831, 0.31093600392341614, 0.4458150267601013, -1.4819377660751343, -0.37312331795692444, -0.0031385147012770176, 0.8908218741416931, 0.04061388224363327, -0.7420862913131714, -0.18495787680149078, -0.26853761076927185, -1.7949334383010864, 0.17029909789562225, 0.804678201675415, 1.088873028755188, 0.8643280863761902, -1.0255829095840454, 1.1838024854660034, -0.5774439573287964, 0.825040876865387, -0.9044594764709473, 1.2819058895111084, -0.6340071558952332, -1.6790752410888672, -0.30019035935401917, 0.726789653301239, 1.217995047569275, -0.27139681577682495, -1.2478663921356201, 0.7315535545349121, -0.42068856954574585, -0.49064791202545166, -0.5583620667457581, 1.6190255880355835, 0.5912901163101196, 0.7632768154144287, -0.6248349547386169, -0.9886300563812256, -0.4974187910556793, 0.57952481508255, 0.8429797291755676, 1.091310739517212, 0.5895249843597412, -1.2223222255706787, 0.10462359338998795, -0.032475367188453674, 2.824216604232788, 0.2555640935897827, -0.11932546645402908, 1.550244927406311, 0.10093463212251663, -0.9527576565742493, -1.573326826095581, -0.9668748378753662, 0.8419747948646545, -0.6329607367515564, -0.7062304615974426, -0.8055530786514282, 1.5196393728256226, 1.150997281074524, 0.21529258787631989, 0.3132922053337097, 0.00770035432651639, 0.6837631464004517, -0.5614344477653503, -0.48944970965385437, 1.5254013538360596, 0.21116219460964203, -0.36506035923957825, -0.5960606336593628, 0.7605865001678467, 1.2773343324661255, 0.07060030102729797, 1.2089121341705322, 0.17058871686458588, 0.783654510974884, -0.26882484555244446, -0.5249021053314209, -1.6692255735397339, 0.024568231776356697, 0.552689790725708, -0.9907404184341431, -1.0612236261367798, 2.488344192504883, -0.1996714025735855, -1.4485785961151123, -0.4379602372646332, 0.3144720792770386, -1.4210035800933838, -0.9467174410820007, -1.8061082363128662, 0.5268366932868958, 1.8197455406188965, 0.9869958162307739, 0.6834547519683838, 0.7720484733581543, -0.8371440172195435, -1.2193337678909302, -0.6714094281196594, -0.6492350101470947, 0.014911573380231857, 1.2143884897232056, -1.7445943355560303, 0.4306398928165436, 1.08144211769104, 0.391250342130661, 3.274109363555908, -0.11592472344636917, -0.12347610294818878, 0.28709492087364197, 0.5745686292648315, -0.5554032325744629, -0.7221463322639465, 0.4436224102973938, -1.1281228065490723, 1.176706075668335, 0.91037917137146, 0.4227796792984009, -2.6599955558776855, 0.6227094531059265, 1.6690888404846191, 1.1471627950668335, -0.28684529662132263, 0.9296939373016357, 1.8385915756225586, -1.6904886960983276, 0.7144758105278015, -0.9518526196479797, 0.29724445939064026, -0.5209309458732605, -0.21983958780765533, -1.1590479612350464, 0.9411762356758118, 0.7372931241989136, -1.203068494796753, -0.19680631160736084, 1.6680819988250732, -0.8048165440559387, 1.1839717626571655, 0.5971212387084961, -0.2866000533103943, 0.9396010637283325, -0.947466254234314, -1.394912838935852, 0.6960771083831787, 0.003568575717508793, -0.16621308028697968, 0.3117204010486603, -0.48347702622413635, 1.1396225690841675, 0.916222333908081, 0.17790311574935913, -0.0114209596067667, 1.1896507740020752, 0.6030295491218567, -1.2961903810501099], [0.49081116914749146, 0.2059839516878128, -0.5681456923484802, 1.6235955953598022, 0.007959508337080479, 1.113197922706604, -0.7554941773414612, 0.7386583685874939, -0.9367689490318298, -0.5866008400917053, -0.256369948387146, -1.6651525497436523, 0.2663877308368683, 0.36096787452697754, -0.8336746692657471, 1.287945032119751, -1.027502179145813, -1.3798842430114746, -0.9527513384819031, -0.8336431980133057, 1.3929071426391602, 0.6758081316947937, -1.314157485961914, -1.0052040815353394, -0.4512612819671631, -0.7537623643875122, -0.015126276761293411, 1.5478184223175049, 0.1999535858631134, -0.009428386576473713, -0.39081212878227234, 0.03975171223282814, -0.3077787458896637, 1.4333806037902832, 1.238479733467102, 1.445887804031372, 0.71180659532547, 1.7312123775482178, 1.934536337852478, -0.1034417673945427, -0.8827288150787354, -1.333411455154419, 1.5080333948135376, -0.42132532596588135, -0.2521047294139862, -0.373506098985672, 0.08647670596837997, 0.12346738576889038, 0.9943887591362, 0.9688075184822083, 0.18933646380901337, -0.3450329303741455, -0.7331922054290771, -1.1076658964157104, 0.6233922243118286, 0.9633170366287231, 0.4329414367675781, 0.44229400157928467, 1.7771022319793701, 1.0655862092971802, -0.22337234020233154, -0.9526174664497375, -0.26646021008491516, -0.01499063428491354, 1.0814265012741089, -1.6408075094223022, -0.1925741732120514, -1.03492271900177, -0.18361850082874298, -0.2828802466392517, 3.03692626953125, -1.2912356853485107, -0.09691685438156128, 0.9261373281478882, 0.19605578482151031, 0.6192272305488586, 1.3285635709762573, -1.3363604545593262, 0.9515798091888428, -0.4103935956954956, -0.12262015789747238, -0.6650042533874512, -1.1215578317642212, 0.3789587616920471, 1.666122555732727, -0.3965604603290558, 1.279640555381775, 1.6899186372756958, 0.6547036170959473, 0.6221922039985657, -1.190201997756958, -0.4123225212097168, 0.3620600402355194, -0.8112065196037292, 1.0928159952163696, -0.8442737460136414, 1.0860344171524048, -0.07533415406942368, -0.41863393783569336, 0.03279099985957146, 0.8887432813644409, 0.47351452708244324, -1.2697899341583252, -1.1977224349975586, 0.8204580545425415, 0.5164402723312378, 0.9370683431625366, -0.304604709148407, -1.0737088918685913, 0.1618598997592926, 0.12216529995203018, 0.41287094354629517, -0.6907979846000671, 0.7674659490585327, 1.462971568107605, -0.3790467381477356, 0.50383061170578, 0.6047176718711853, 0.01703924499452114, -0.8234022855758667, 0.08620014041662216, -2.0653188228607178, 0.3449143171310425, 1.0924752950668335, 1.2060692310333252, -0.2907385528087616, -0.16556741297245026, 0.7056041955947876, -0.907755434513092, -0.09907294809818268, -1.50761079788208, -1.1584129333496094, -0.09115930646657944, -1.8952643871307373, -1.1039806604385376, 1.668408751487732, -1.575020432472229, -0.8952255845069885, -1.2350976467132568, 1.2089413404464722, -0.7913846373558044, -0.48112931847572327, 0.3256182074546814, -1.2193782329559326, -0.5444865822792053, 0.7106008529663086, 1.5782898664474487, -0.287698358297348, 0.36593878269195557, -0.21659666299819946, 1.306351661682129, 1.7815055847167969, 0.6146411299705505, -0.5547780990600586, 2.0677120685577393, 0.1894259750843048, 0.44270217418670654, -0.5771583914756775, -0.09653590619564056, -1.2042770385742188, -0.03381114825606346, -0.5894752740859985, -1.7417457103729248, -0.8616530895233154, -0.0034927099477499723, -0.3699062168598175, -0.732807993888855, 0.15319298207759857, -0.13991162180900574, 0.6067674160003662, -0.6403900980949402, -1.8285163640975952, 1.1181669235229492, 1.2735822200775146, 1.056543231010437, 0.6944487690925598, 1.1297540664672852, -1.6260756254196167, 1.0162556171417236, -1.7120115756988525, 1.0661855936050415, 0.6266621947288513, -0.3442506194114685, 0.8214417695999146, 1.0232765674591064, -0.3543435037136078, -0.5626461505889893, 0.15689679980278015, -0.3801157772541046, 1.8552688360214233, 1.3392599821090698, -0.9881536364555359, 0.3830759525299072, 0.2927948832511902, 0.04960372671484947, 1.0003376007080078, -0.5918852090835571, -0.7689179182052612, 0.12092496454715729, 1.523830771446228, -2.4346020221710205, 0.7597690224647522, -0.7961978912353516, 2.238513231277466, -1.8114242553710938, -0.3795192837715149, -0.5370309352874756, 1.0142607688903809, -1.4756895303726196, -1.0944006443023682, 0.4068359434604645, -3.53747820854187, 0.10469504445791245, -1.830163598060608, -0.051562607288360596, -1.2655812501907349, -0.18387307226657867, -0.18602398037910461, 0.6017047762870789, -0.5480796098709106, 2.6923460960388184, -2.3379788398742676, -0.6398639678955078, -0.4317609369754791, 0.3639999330043793, 0.8593984842300415, -0.22897981107234955, 2.1452207565307617, -1.641601800918579, 1.8097199201583862, -0.0749955102801323, -1.045137882232666, -1.3061479330062866, 0.1328168660402298, 0.4242357015609741, -0.007990917190909386, -1.156518578529358, -0.7938327789306641, 1.9042028188705444, 1.415525197982788, -0.7560453414916992, 0.1477639377117157, -0.11115127056837082, -0.6641435027122498, -0.28849637508392334, -2.1382641792297363, -1.3656507730484009, -1.3744560480117798, -1.178431749343872, -0.11807824671268463, 1.6672048568725586, 0.21929271519184113, -1.2881795167922974, 1.3635262250900269, 0.024848226457834244, -0.005558462347835302, -0.45322585105895996, -0.25252294540405273, -0.5837856531143188, 1.7344974279403687, 0.37939152121543884, -0.12860505282878876, -0.1401883363723755, -0.5638709664344788, 0.4036842882633209, 1.2432600259780884, -1.1131526231765747, -1.2064838409423828, 0.5537457466125488, 0.43103310465812683, -0.5668905377388, 1.9760780334472656, -0.21213525533676147, -1.138444185256958, -0.5324628353118896, -0.18010620772838593, -0.0661235973238945, 1.552417516708374, -0.04128806293010712, -1.2081644535064697, -0.5587597489356995, -0.4912039339542389, 1.194952368736267, -2.370163679122925, 0.5420742034912109, 1.3683985471725464, 0.6165936589241028, 0.3380185067653656, -0.14681653678417206, -1.2529133558273315, -0.7145436406135559, 1.7521380186080933, -2.144361972808838, 0.910234808921814, 0.1829090118408203, 0.38766294717788696, -0.20568756759166718, 0.939769983291626, -0.9818938970565796, 0.14164380729198456, -0.7366699576377869, 0.6300525069236755, -0.662798285484314, 1.6537624597549438, -0.016277331858873367, -0.46286484599113464, 2.0905842781066895, 0.7712005376815796, 0.27754300832748413, 2.3780877590179443, 1.4342015981674194, 0.3686459958553314, 1.0400350093841553, 0.5032714605331421, 0.5588395595550537, -0.8227093815803528, 0.13649219274520874, -0.2823892831802368, 0.9133908152580261, -1.9168275594711304, -0.14011314511299133, -0.757914125919342, 0.6495028734207153, 0.532983660697937, 0.9825379252433777, 0.171285942196846, -1.7132878303527832, 0.031434912234544754, -0.9326721429824829, 0.9141982197761536, 0.5949254631996155, 1.1101356744766235, -0.5895427465438843, -1.3508636951446533, 0.5809955596923828, 0.8304027318954468, 2.5679948329925537, -1.0193028450012207, 0.7214649319648743, -0.07130547612905502, -1.1703709363937378, 0.17330272495746613, -0.2664666175842285, 0.6216599345207214, 0.9170969128608704, -1.9179927110671997, -2.0858957767486572, 1.700527310371399, -0.12521588802337646, -0.2410094439983368, 1.0217955112457275, -0.22195148468017578, 1.9022973775863647, -2.71201753616333, -0.4562859535217285, 1.194056749343872, -1.0166746377944946, -0.27502936124801636, -0.6443206667900085, 0.6479068994522095, 0.6976077556610107, -0.4565589427947998, 2.1222596168518066, 2.5950071811676025, 1.5842710733413696, 0.926658034324646, -0.30474773049354553, -0.6966421008110046, 0.34119895100593567, -0.9272960424423218, -1.599242925643921, -0.37841174006462097, -0.24378514289855957, 1.0636988878250122, 0.1977931559085846, 0.9919031262397766, 0.5347357392311096, -0.61570143699646, 0.8087719082832336, 0.26490387320518494, 0.7592958211898804, -0.3602771461009979, 0.4197964668273926, 0.8276242017745972, -1.4977585077285767, -1.343571662902832, -1.6448813676834106, -0.999981701374054, 1.7137031555175781, -0.9119598865509033, 1.361680269241333, -1.3258540630340576, -0.5412058234214783, 0.36798620223999023, 0.24153247475624084, -1.2776988744735718, -0.878286600112915, 1.4025795459747314, -0.10772839188575745, -1.3329105377197266, 0.14287294447422028, -1.9532968997955322, 1.256686806678772, 1.4875688552856445, 0.3031940460205078, 0.8364267945289612, 0.9216675162315369, 0.3453117907047272, 0.3914693295955658, -1.4681411981582642, -0.6467186212539673, 0.31274762749671936, -0.17738912999629974, -0.3715236783027649, 0.3332660496234894, -0.06781090795993805, 0.6327719688415527, -1.1842788457870483, 1.076388955116272, 0.6569843292236328, 1.1055923700332642, -0.9519933462142944, 2.075815200805664, -1.6390528678894043, 1.0183030366897583, -2.09130597114563, -0.0930584967136383, 0.7621335983276367, 0.9037870764732361, 1.6948517560958862, -1.7499186992645264, 0.44034647941589355, 0.19422340393066406, 0.45507532358169556, -0.47520941495895386, -0.005920564290136099, 0.9265603423118591, -0.660550057888031, -0.5472019910812378, -0.42899832129478455, 1.1467825174331665, 0.8475521802902222, 0.5655997395515442, -0.2030480057001114, -0.597421407699585, 0.24734574556350708, -1.0348894596099854, -0.43448442220687866, 2.1508102416992188, -0.05963249132037163, -0.34832853078842163, -0.008001812733709812, -0.09131252020597458, 1.3937879800796509, -1.4212521314620972, -0.03677909076213837, 0.08939659595489502, -0.33608976006507874, -0.09359120577573776, 2.350111961364746, -0.47000283002853394, 0.030128605663776398, 1.4405566453933716, 0.020294571295380592, -0.3528610169887543, 2.427138328552246, 0.8757302165031433, 0.1286272406578064, -0.6016964912414551, -0.37014007568359375, 1.7499396800994873, -0.4562170207500458, -0.09884726256132126, -0.5033599138259888, 1.8430825471878052, 0.9889122247695923, -0.6635124087333679, -1.4782652854919434, -0.9738689661026001, -1.8196611404418945, -0.7339842319488525, -0.12613670527935028, 0.39211568236351013, -0.05067981779575348, 0.8509969711303711, -0.05199159309267998, -1.3416838645935059, 0.7450928688049316, 0.6101385354995728, -0.2860702872276306, 0.115322045981884, 0.5599727630615234, -1.1131325960159302, 0.7249574065208435, -1.862009882926941, 0.5558458566665649, 0.6691392064094543, 1.786171793937683, -0.7527796626091003, 0.13551248610019684, 0.42601701617240906, -1.8561217784881592, -0.6028640270233154, -0.40514668822288513, 0.4370594024658203, 1.2751538753509521, 0.006335721351206303, -1.002018690109253, -0.7164451479911804, -1.1766798496246338, -0.7546272873878479, 2.356078624725342, 0.45931217074394226, 0.23947298526763916, -0.12176090478897095, -0.12071491032838821, -0.8702335953712463, 0.5998882055282593, -0.174711674451828, -0.29501113295555115, -0.409955769777298, 0.591931939125061, -0.47368210554122925, 1.5234153270721436, -0.973228931427002, 1.4253363609313965, 0.17214298248291016, -0.011957095935940742, -0.2929478585720062, -0.9795671701431274, 0.57551509141922, 0.47967278957366943, -0.18036650121212006, 0.307833194732666, 0.6417604684829712, -0.07575611025094986, -0.8790683746337891, -1.2586661577224731, 0.8752238750457764, 0.6834847927093506, 0.5811544060707092, -1.6353554725646973, -0.24908447265625, 0.505739688873291, 1.3108137845993042, -0.8357751965522766, 0.1718064695596695, -0.09658309072256088, 1.2710191011428833, 1.0362827777862549, -1.3217875957489014, -0.7188998460769653, -0.031160056591033936, 1.7084393501281738, 0.10807090997695923, -0.7481052279472351, 0.3261941373348236, -0.8181003332138062, 0.8243355751037598, -1.8780828714370728, -0.2842195928096771, -1.01925528049469, 0.6387273073196411, -1.0642435550689697, -0.024326656013727188, 1.9007787704467773, -1.1633795499801636, -0.2770366668701172, -0.8005170822143555, 1.1244611740112305, 0.6467369794845581, -0.6177033185958862, 1.0393168926239014, -0.5900285840034485, 0.771469235420227, 0.3656958341598511, 0.9053781628608704, -0.6904487609863281, 0.5262141227722168, -0.724568247795105, 1.6549931764602661, -0.5229693055152893, -0.6491788029670715, -0.831663191318512, -0.4694163501262665, -0.821785032749176, -0.024211475625634193, -0.1798761487007141, -0.605824887752533, -2.0485947132110596, -0.39548641443252563, -0.6730214953422546, -0.6587201952934265, 0.0638192743062973, 0.982673704624176, -0.605884313583374, 1.3819060325622559, -1.9990441799163818, 1.861817479133606, 0.8778157830238342, -1.2113451957702637, 0.18653741478919983, -0.060728441923856735, 0.5033186078071594, 0.05557220056653023, -0.12840549647808075, 0.3838321566581726, 0.38818421959877014, 0.14808444678783417, 1.8465009927749634, -1.408166766166687, -0.7246716022491455, 0.46014532446861267, 0.3619181215763092, -1.4758343696594238, 1.318981647491455, 0.7651064395904541, -0.6161399483680725, 1.356057047843933, -0.9778480529785156, -0.6415674686431885, -2.1496269702911377, 1.9607635736465454, 1.7815107107162476, 0.8954344391822815, 0.5923834443092346, 0.4313102066516876, -0.8042306900024414, -0.21527209877967834, 0.5900283455848694, -0.0695602297782898, -0.8928895592689514, 1.4844862222671509, 0.9377374053001404, -1.9826089143753052, -0.7672035098075867, -0.23029610514640808, -0.5223168730735779, -0.07179556041955948, -1.3393142223358154, -0.6103353500366211, 0.03583510220050812, 1.2459723949432373, -1.9744869470596313, -1.360257625579834, -0.022678906098008156, 0.4881875216960907, -0.22096385061740875, 0.7838689684867859, 1.5372365713119507, -0.6598456501960754, 1.0135327577590942, -0.3370673954486847, -0.49991241097450256, -0.273354709148407, 0.26798170804977417, -0.34204792976379395, 0.136265367269516, 0.6142212748527527, 0.9771830439567566, -1.457192301750183, -0.33117055892944336, 0.07475665211677551, -0.28405776619911194, -0.003505678614601493, -0.19289036095142365, -0.39095792174339294, -1.6374930143356323, 1.0251216888427734, 1.0405852794647217, -1.2573165893554688, 1.5754307508468628, 0.49332454800605774, 1.4547462463378906, 0.05069933459162712, 0.7803115248680115, -0.005987819749861956, -1.4784127473831177, -0.11966858804225922, -0.9137271642684937, -1.6291511058807373, -0.6549538969993591, -0.008110896684229374, -0.9669134616851807, 0.9188896417617798, 1.1242790222167969, 1.5104809999465942, -1.409525752067566, 0.22898393869400024, -0.44539520144462585, -1.0057023763656616, 1.6717512607574463, 0.8617778420448303, 0.7286559343338013, -0.000346166780218482, -1.1280627250671387, 2.082350015640259, -0.5122102499008179, 0.6399221420288086, 0.1459416002035141, 1.3306301832199097, 0.1365356296300888, -1.4119640588760376, -1.5575491189956665, 0.6453174352645874, -0.21126940846443176, -0.82940274477005, 0.24379628896713257, -1.1349455118179321, 0.36680030822753906, -0.9067937731742859, -1.3585326671600342, 0.3764893710613251, 0.5949968099594116, -0.40778428316116333, 0.6128380298614502, -0.6424809098243713, -0.24261806905269623, -0.5820517539978027, 0.13808466494083405, 2.454221725463867, 1.2373989820480347, -1.852936863899231, -1.311432123184204, -0.22019873559474945, 1.3494107723236084, -1.2208967208862305, -0.7519522905349731, 0.6193616390228271, -0.3818514943122864, -0.7039605975151062, -0.22710564732551575, 1.428259253501892, 0.1571553349494934, -0.47532135248184204, -1.3181922435760498, 0.32592377066612244, -1.1505852937698364, -0.44612374901771545, -0.8210041522979736, 0.505979597568512, -2.528425931930542, -0.7502447366714478, 0.5439925193786621, -0.5890650749206543, 0.5069804191589355, 1.3287997245788574, 0.18968823552131653, 0.6148732900619507, 0.20444855093955994, -0.653098464012146, -0.3252430856227875, -2.1469674110412598, 2.088503360748291, 0.09121514856815338, 0.42090973258018494, 0.7759525179862976, -0.9250930547714233, -1.3545933961868286, 0.17522788047790527, -1.331680417060852, -1.5330991744995117, -0.39588725566864014, -1.423108458518982, -0.09650163352489471, 0.4872708320617676, -2.4033567905426025, 0.5945283770561218, -0.2688663601875305, -0.6806164979934692, -0.9977476000785828, -0.09551803767681122, -1.1553205251693726, -0.3708757162094116, 0.12928655743598938, -0.04401685670018196, -0.2500236928462982, 0.3737427592277527, -0.5851352214813232, -0.3603270649909973, 1.9844242334365845, 1.0518102645874023, -0.8488963842391968, -0.3816658556461334, -0.5928590893745422, 0.4994766414165497, 1.3065797090530396, 0.4240768849849701, -0.49872303009033203, -0.09484364092350006, -0.37285956740379333, -0.3329312205314636, -0.7498974204063416, -1.5041109323501587, 1.1161324977874756, 1.6683834791183472, -0.10725235939025879, -0.8855427503585815, -0.42114484310150146, -0.22122257947921753, 0.2455173283815384, 1.0233166217803955, 0.9898285865783691, -0.7953981757164001, 1.0732529163360596, -0.029978148639202118, 0.5645372271537781, -0.8661072850227356, 0.3206327557563782, -1.4557969570159912, 1.6488481760025024, -0.19446691870689392, -1.4950008392333984, 0.9485366940498352, 3.4193975925445557, 0.5232025384902954, 0.47860580682754517, 0.6523634791374207, 0.1995731145143509, -0.8020352721214294, 0.4863332509994507, 0.2170036882162094, -1.050311803817749, -0.7593781352043152, 1.2800666093826294, -0.09077087044715881, -0.17145095765590668, -0.18945454061031342, 0.411742240190506, 0.520803689956665, 1.4147547483444214, -1.453410029411316, 1.450226068496704, -2.610328435897827, -0.9558242559432983, -0.22171416878700256, -1.3817789554595947, -1.0266720056533813, -0.12756206095218658, -0.2702341675758362, -1.037265419960022, 1.2561273574829102, 1.0304532051086426, -1.6211819648742676, 0.5721908807754517, -1.8761919736862183, 0.9305374026298523, 1.160602331161499, -0.14976368844509125, 0.02292981930077076, -0.1988731026649475, 0.4703546166419983, 1.5183392763137817, -0.1406942903995514, -0.2495153397321701, 0.7706037163734436, -0.36567258834838867, 0.38974452018737793, -0.08959785103797913, -0.38708406686782837, 1.2984583377838135, -0.8618667721748352, 1.8181923627853394, -0.39749330282211304, 0.9607752561569214, 0.2749589681625366, -0.21802082657814026, 0.9192286133766174, 0.6849938631057739, -0.4504428207874298, 1.5761222839355469, -0.4034408926963806, 1.9767253398895264, 1.82052481174469, -0.6746993064880371, 1.3345510959625244, -1.5658413171768188, 0.7408949136734009, 0.25679340958595276, 0.8526323437690735, 0.6339276432991028, -0.5673785209655762, -0.7923306822776794, -0.059925567358732224, 0.5098897218704224, 0.4018018841743469, -0.026023296639323235, 0.4084288775920868, 0.027098054066300392, -0.3201836943626404, -0.42583614587783813, 0.4483431875705719, -1.8084070682525635, 1.0805010795593262, 0.9604419469833374, -0.4909522533416748, 0.4794042706489563, -0.11068779975175858, -0.480485737323761, -1.5549973249435425, -0.2623085081577301, -0.6370761394500732, -0.6026861667633057, -0.26462623476982117, 0.3348044157028198, 0.46740734577178955, 0.2305997759103775, -1.074052095413208, 0.19501513242721558, 0.7478548884391785, -1.9669311046600342, -1.446886420249939, -1.9684147834777832, -1.3652167320251465, -0.619733452796936, -0.09080187231302261, 0.2235449105501175, -0.5248046517372131, 2.3165249824523926, -0.8935603499412537, -1.0655410289764404, -1.0704920291900635, -0.5642710328102112, -0.7721073627471924, 0.6717917919158936, 0.596143901348114, 1.235986351966858, -0.18135708570480347, -0.33078503608703613, 0.34825780987739563, 0.5261865258216858, -0.05229216441512108, 0.07605566829442978, -0.12200244516134262, -0.10664710402488708, 1.27599036693573, -0.9981300234794617, -0.08075518906116486, -1.945727825164795, 0.2960236370563507, -0.14868460595607758, 0.8073216080665588, 1.5575381517410278, -1.0649341344833374, 0.5913162231445312, 1.1440818309783936, -0.9267089366912842, 1.7170454263687134, 0.4571293890476227, -1.4896548986434937, 0.0845165029168129, 0.10888313502073288, -1.7553902864456177, 1.5520066022872925, -0.8576819896697998, 1.8693112134933472, -0.14551815390586853, -1.7164584398269653, 1.7509980201721191, -0.34171706438064575, 1.4752641916275024, -0.6179355978965759, 1.371206283569336, 0.5962921977043152, -1.448775053024292, -0.49777454137802124, -1.0102812051773071, 2.099402666091919, -1.2604475021362305, 0.6170327067375183, -1.0318511724472046, 0.10847648233175278, 0.5469563603401184, 0.3569309115409851, 0.9627203345298767, 0.3197953701019287, 1.1403441429138184, 0.7350451946258545, -0.25644269585609436, -0.11575858294963837, 1.3963688611984253, -0.6677092909812927, -0.9278795123100281, -0.5947533249855042, -1.4733974933624268, 1.3030232191085815, -0.730567991733551, -0.5399619936943054, -2.056757926940918, 0.006826506461948156, 0.19895336031913757, -0.04643343389034271, 0.3483614921569824, 0.0022585263941437006, 0.17103122174739838, -0.1349189281463623, 1.11782968044281, -1.3425734043121338, 0.7018284797668457, 0.37475350499153137, -1.8103348016738892, -0.9510676860809326, 0.8249413371086121, 0.5662486553192139, 0.7977682948112488, 1.7841718196868896, 0.11670709401369095, -0.019325198605656624, -0.2586551904678345, 1.4737575054168701, 0.7296715974807739, 0.3188880383968353, 0.4108024835586548, -2.807243824005127, 0.355602890253067, -0.687519371509552, 1.3970568180084229, -0.48714837431907654, -0.7544307708740234, -0.8332944512367249, -0.4913036823272705, 0.055130280554294586, 0.9024182558059692, -1.1162998676300049], [-0.3520277142524719, -1.2583357095718384, -0.20705150067806244, -1.0302852392196655, -0.8461176156997681, -1.4856407642364502, 1.108302116394043, -0.3140268325805664, -0.05643840506672859, 1.0130378007888794, -1.3231086730957031, -1.455065131187439, -0.5637104511260986, -0.445465624332428, -0.11416751146316528, 0.23731355369091034, 0.9228394627571106, 0.3419337868690491, 0.7088976502418518, -0.5105209350585938, 0.7011888027191162, -1.316346287727356, 0.15671305358409882, -1.6972991228103638, -0.5109795928001404, 0.07828590273857117, -1.1696826219558716, -0.6482985615730286, 1.5361603498458862, 0.549185037612915, 0.16057023406028748, 1.6214001178741455, 0.06626113504171371, -0.28015831112861633, -1.4761115312576294, -1.5847718715667725, 0.6515129208564758, 0.913927435874939, 0.759810745716095, 0.7858921885490417, -0.3792000114917755, -0.7098791003227234, -0.92227703332901, 1.503627896308899, -0.6420579552650452, -1.6912813186645508, 0.519801914691925, 1.0904954671859741, 1.452158808708191, -0.1676110476255417, -0.7836688756942749, -0.4393962323665619, 2.317544937133789, -0.49085986614227295, -0.2733858525753021, -0.24328194558620453, 0.7691754102706909, -0.721580982208252, 1.1986050605773926, 0.9390236735343933, -1.8967680931091309, -1.284792184829712, 1.582979440689087, -1.2592154741287231, -1.162731409072876, 0.614857017993927, -0.207395538687706, 1.073957085609436, 1.0081788301467896, 1.8642886877059937, 0.04778650030493736, 1.1927851438522339, -0.014025036245584488, 0.08707141131162643, 0.7655881643295288, -0.19284206628799438, -0.6138703227043152, 0.02480301633477211, 1.1797641515731812, 0.7799428701400757, -0.08243820071220398, 0.9595730900764465, 0.048327088356018066, 1.0955772399902344, 0.07063403725624084, 0.27321189641952515, 0.7635380625724792, -1.480575680732727, 0.3469279110431671, -0.9107524156570435, 0.1294792741537094, 0.09993773698806763, 0.3615853190422058, 0.19008497893810272, -1.715686559677124, -0.9912916421890259, -1.0577324628829956, 0.3652920126914978, 1.024599552154541, -0.2826162576675415, -0.1665685623884201, -0.19161158800125122, -1.2442256212234497, -1.5483372211456299, -1.1008800268173218, 1.597718358039856, 0.8094345927238464, 0.8093098402023315, 0.13240492343902588, -1.1043179035186768, -0.3009132146835327, -1.8972153663635254, 0.38394683599472046, -1.476279616355896, 0.46468156576156616, -0.23099391162395477, -0.24175158143043518, 1.3125375509262085, 0.6327138543128967, 0.3826148211956024, 0.8234521746635437, 1.4823435544967651, 0.5995141863822937, -0.42211684584617615, -0.5356108546257019, -1.4030359983444214, 0.29856595396995544, 0.8084896802902222, 1.5096614360809326, -0.856418251991272, 0.551655113697052, 1.3257049322128296, -0.5870793461799622, -0.30485644936561584, -0.6862671375274658, 1.4028359651565552, -1.5311810970306396, -0.4864766001701355, -0.7014816999435425, 1.1722735166549683, 1.6820961236953735, 0.09948547184467316, -1.3616199493408203, 0.25739356875419617, -0.5881335735321045, -1.1843843460083008, 1.4009000062942505, 0.7207188010215759, -0.7558100819587708, -1.6664848327636719, 0.3960763216018677, -1.479207992553711, 2.0203471183776855, -1.3951241970062256, -0.8720855712890625, -0.27357929944992065, 1.013142704963684, 0.9484560489654541, -0.8648212552070618, 0.7290195822715759, -0.15046754479408264, 0.24961359798908234, 0.2569323182106018, -0.7302104234695435, -0.5396561026573181, -1.724298119544983, 0.3421059250831604, 0.25366947054862976, 0.09680870175361633, 0.05874983221292496, 0.5945924520492554, -0.3366199731826782, -0.6718604564666748, 1.190663456916809, -0.007678277790546417, 1.4331717491149902, -0.3318674862384796, -1.1702460050582886, -0.6567036509513855, 0.7528588175773621, 1.3090428113937378, -0.1881254017353058, -2.240687370300293, 1.6771430969238281, -0.36163341999053955, -2.0629079341888428, 1.5177761316299438, 1.018388032913208, 1.1591624021530151, -0.4896300137042999, -0.9995459318161011, 0.376606285572052, -0.061730045825242996, 1.6961019039154053, -0.2800683379173279, 0.34633904695510864, 0.4485257565975189, 1.8983348608016968, 0.5360971689224243, 0.6201446652412415, 1.5494388341903687, -0.11185912787914276, -1.3981062173843384, -0.034716006368398666, 0.3445206582546234, -1.2412409782409668, 0.4270283877849579, 0.6633419990539551, -0.5091286897659302, 0.49468445777893066, 0.7151150107383728, 1.2574961185455322, 0.877376139163971, -0.7511997818946838, -2.828153133392334, -1.4951298236846924, 0.845568835735321, -0.7906092405319214, -1.4783276319503784, -0.6585744023323059, 0.6943334341049194, -0.8880552053451538, 0.498523086309433, 0.5531585812568665, 0.32972487807273865, 0.887370765209198, 0.11550450325012207, 0.15150059759616852, 0.6176421046257019, 0.8888188004493713, 0.3983885943889618, -1.3766850233078003, 0.24769572913646698, -0.7466145157814026, -0.6025286912918091, 0.2640807032585144, 0.3567301332950592, 0.3750476837158203, -0.6496537327766418, -0.483102023601532, 1.1491488218307495, 0.5685570240020752, -0.7400438785552979, -1.354204773902893, 1.175331711769104, 0.9505416750907898, -2.892768144607544, -0.31485453248023987, -0.3243030309677124, -1.0961562395095825, 2.2267491817474365, -0.929326057434082, -0.8106390833854675, -0.09703042358160019, 1.103240966796875, -2.106511116027832, -0.3343857228755951, -0.3014060854911804, 0.8135924339294434, -0.6214967966079712, -0.2921750545501709, 1.5015695095062256, 3.260408639907837, 0.08407483994960785, -2.1292498111724854, 1.3886427879333496, 1.7753331661224365, -0.7756451368331909, 0.1300540566444397, 0.5128166079521179, -0.17725542187690735, -1.5137966871261597, 0.809754490852356, -0.3821978271007538, 0.10473917424678802, 1.6850959062576294, 0.7031407952308655, 0.47448787093162537, 1.308660864830017, 1.4334510564804077, 1.115343689918518, -1.3360295295715332, -0.7087045907974243, -0.538378119468689, 0.7554289102554321, -0.3697809875011444, 1.956016182899475, 0.9350404739379883, -0.4025256633758545, 0.1330515295267105, 0.9947167634963989, -0.7267075181007385, 1.5161434412002563, -0.15419764816761017, -0.4157275855541229, 1.727044701576233, 0.08934419602155685, -0.6063411831855774, 0.611656904220581, -1.0694184303283691, -1.244051218032837, -0.5786007046699524, 0.14222364127635956, 0.08013427257537842, 0.49607956409454346, -0.13652676343917847, 0.36486393213272095, -0.7931314706802368, -0.3234710693359375, 0.012467525899410248, -0.7063226103782654, 0.39404353499412537, 0.660412073135376, -1.2118055820465088, -0.13745155930519104, -1.5899360179901123, 0.43883657455444336, -1.2247644662857056, -0.9771832823753357, -1.610561490058899, -0.22789569199085236, 0.8827994465827942, 0.3878606855869293, -0.42808106541633606, 0.22639648616313934, 1.8407461643218994, -1.3795640468597412, -0.4533851742744446, -0.9242652654647827, 0.11444885283708572, -0.3983989655971527, -1.1518428325653076, -2.0463311672210693, -0.25992104411125183, 1.19588041305542, -0.8876117467880249, 0.6725080013275146, 1.2137559652328491, 2.077824831008911, -0.4322185814380646, -0.1675311028957367, -1.165431261062622, -0.04370059445500374, 0.5898159742355347, -0.45504626631736755, -0.05153517797589302, 0.0099709527567029, -0.39486953616142273, 1.1500645875930786, 0.4004298746585846, -1.4129079580307007, -0.3572227656841278, 1.0317697525024414, -1.0077444314956665, -1.1935697793960571, 0.6143603324890137, 1.323621392250061, 0.28405386209487915, 0.6129602789878845, -0.3079661428928375, 0.049025051295757294, 1.6575069427490234, 0.31466880440711975, -1.7630926370620728, -1.2673524618148804, 1.1070339679718018, 0.2996846139431, -0.5336915254592896, 0.4500831067562103, 0.6414545178413391, 1.324326515197754, 0.4480234384536743, 1.9982707500457764, -1.6030808687210083, 0.9062743782997131, 0.22433586418628693, 0.7899774312973022, -1.8731532096862793, 0.037305254489183426, 1.1071195602416992, 1.9551160335540771, -0.06335145980119705, -1.2999929189682007, -1.6081331968307495, 0.20243233442306519, 0.40527108311653137, -0.46854308247566223, -1.7055681943893433, 1.2622612714767456, -1.3737127780914307, 0.5932214260101318, -0.4460194706916809, -0.7757850885391235, 1.4561179876327515, 0.0028020679019391537, -2.0665457248687744, -0.6531482338905334, 1.857162356376648, -0.15826350450515747, 0.848416268825531, -1.2243366241455078, -0.011908532120287418, 1.3595818281173706, 0.19122248888015747, -1.0694721937179565, -0.7775773406028748, 0.0027517168782651424, -0.7438675761222839, 2.790571451187134, -0.543993353843689, 0.2684643864631653, 0.812597930431366, -0.9143417477607727, -0.8638327717781067, -0.0273037888109684, -0.03630223870277405, 0.4751379191875458, -2.331331491470337, -0.6218462586402893, 0.5674772262573242, -0.20396064221858978, -0.5927377343177795, 0.9032217860221863, -0.045987196266651154, -1.064139485359192, -0.7833678722381592, 0.11020751297473907, 0.33411353826522827, -1.3024797439575195, -0.026368500664830208, -1.3364181518554688, 0.9171376824378967, 0.9694734811782837, 1.1027580499649048, 1.22354257106781, -0.405121773481369, -0.4962335526943207, -0.09830233454704285, -0.47303515672683716, 0.09449844062328339, 0.5385329127311707, 1.8777062892913818, -0.25449416041374207, 0.18936733901500702, -0.7385923266410828, -0.9654552936553955, -0.2784171998500824, 1.245843529701233, 0.5349368453025818, 1.4831138849258423, 0.1865815371274948, 1.276483178138733, -1.5427101850509644, -1.4280034303665161, -0.675804615020752, -0.9518918395042419, 0.35372239351272583, 0.544477105140686, -1.2866058349609375, -0.3017829954624176, 1.4707050323486328, -0.009115547873079777, -0.06701447069644928, 0.47723060846328735, -1.8167909383773804, 0.09835740178823471, 0.11850257962942123, -0.3369966447353363, -1.5039170980453491, 1.5067455768585205, -0.6107929944992065, -0.3380354046821594, -0.392939031124115, -0.1052631139755249, -1.2788028717041016, -0.23737286031246185, 0.9129772782325745, 1.3398241996765137, -1.3915756940841675, 0.4187375605106354, 0.8208869695663452, -0.22847342491149902, 0.4914850890636444, 1.9078881740570068, 2.4957191944122314, 0.23170967400074005, 0.510216474533081, -0.7772656679153442, -0.005032369401305914, 1.4896539449691772, -0.5569578409194946, -1.156253695487976, 0.41938337683677673, -0.4007842242717743, -0.7298731803894043, -0.21312013268470764, 0.5346479415893555, -0.2470620721578598, 0.14914187788963318, -0.39173048734664917, 0.2754932940006256, 0.5889116525650024, -1.5488358736038208, 0.5132411122322083, 0.09695746749639511, 0.2610066533088684, 0.8997968435287476, 0.5931627154350281, 0.7400678992271423, 0.751504123210907, 0.7314409017562866, -0.5475393533706665, -0.402649849653244, -0.6767917275428772, 0.6165407299995422, -0.35915595293045044, -0.4982689917087555, 0.04513232037425041, -0.16038808226585388, 0.06716348975896835, -0.5531390905380249, -0.36504411697387695, -0.21960027515888214, -0.3125695288181305, 0.5508460998535156, 0.09376486390829086, 1.0381274223327637, -0.3826787769794464, 0.10148485749959946, 0.011057984083890915, -1.5793170928955078, -0.7459511756896973, 0.7342434525489807, -0.6249915957450867, -0.19849848747253418, 2.8289949893951416, 0.8290376663208008, -0.1299983561038971, 0.011026562191545963, -0.2139834463596344, 0.8013312816619873, -0.9623372554779053, -0.6823736429214478, 1.278751015663147, 0.5727251768112183, 0.8301436305046082, -1.6313121318817139, -0.6708227396011353, -1.8355785608291626, 1.0273442268371582, -1.026788592338562, -0.20015282928943634, 0.8850876092910767, 0.6492047905921936, 0.23093336820602417, 0.3135973811149597, -1.3974268436431885, -0.7319590449333191, 0.8386571407318115, 0.8281007409095764, -0.6072729825973511, 0.41584622859954834, -0.32603827118873596, 1.4153205156326294, 0.6306734085083008, 2.1369969844818115, -0.3100948929786682, 1.3661413192749023, -0.8796584606170654, -0.0042860484682023525, 0.41196349263191223, -0.4482862055301666, 1.06557297706604, -0.480831503868103, -0.832000195980072, 1.637689471244812, 0.28462955355644226, -1.2154898643493652, -0.31610846519470215, -1.209943175315857, 0.5602448582649231, 1.914152979850769, -0.3811410665512085, 0.34786146879196167, 1.2893269062042236, 1.5068892240524292, 0.8874596357345581, 0.015290034003555775, 2.0751278400421143, 2.2811062335968018, 0.6537748575210571, -0.3863082230091095, 0.15805834531784058, 1.224149227142334, -0.6247788667678833, -0.15394428372383118, -2.163705587387085, -0.28892162442207336, -0.29214760661125183, -0.3844311237335205, -1.1262102127075195, -0.5543921589851379, -1.4010933637619019, -0.3011203110218048, -1.0918326377868652, -1.4253729581832886, 0.18204796314239502, 0.3763675093650818, -0.6112532615661621, 1.4511537551879883, 0.030407840386033058, -0.579867422580719, 0.44915667176246643, 0.13062824308872223, -0.5405948758125305, -0.30708038806915283, -0.4788624942302704, 0.24587717652320862, 0.6109638214111328, -0.2564484477043152, -1.5097075700759888, -1.3481850624084473, -0.19710618257522583, -0.9964554905891418, -0.43419548869132996, -0.7762296795845032, -0.5936392545700073, -1.2590136528015137, 1.2926502227783203, -0.9878444671630859, -1.0347096920013428, -0.6151688098907471, -0.28073054552078247, -0.4687199890613556, 0.12160996347665787, -0.24258285760879517, 0.4446250796318054, -0.2402678281068802, 0.233357235789299, -0.1646878719329834, -1.6609327793121338, -1.6524865627288818, -1.8025496006011963, 0.2588063180446625, 0.9491968154907227, -0.7273514866828918, 0.33343008160591125, 0.6366565227508545, 0.24690619111061096, -0.07200499624013901, -1.4753671884536743, 0.6214414834976196, 2.5869765281677246, 1.0823622941970825, 1.2612264156341553, 0.08145853132009506, -0.5351144671440125, 0.8512059450149536, 0.8226726651191711, 0.07345518469810486, -0.4170582592487335, 2.0084285736083984, -1.019656777381897, 1.3908013105392456, -1.7982507944107056, 0.8214974999427795, 0.07129000127315521, -1.4376200437545776, -0.01517221238464117, 0.8056747913360596, 1.3120933771133423, 0.7241021990776062, -1.0749980211257935, -0.8189553618431091, 0.6967635154724121, -0.4862652122974396, -0.4307191073894501, 1.807812213897705, -0.4423309862613678, 0.2700415849685669, -0.9036706686019897, -0.40395477414131165, 0.9976999759674072, 1.727221131324768, 0.5439695119857788, -0.8321763277053833, -0.0895330011844635, -0.4037844240665436, 0.0399075411260128, -1.8945038318634033, -0.3564339578151703, -0.07169690728187561, 0.8448914289474487, -0.977918803691864, 0.7061772346496582, 0.48146238923072815, 0.47127750515937805, 0.4398787021636963, -1.664359450340271, -0.06436536461114883, 1.1189409494400024, -2.205465078353882, 0.45051053166389465, 0.6418914794921875, 0.3082130551338196, -0.11616075038909912, -0.31973183155059814, 0.7303066253662109, 0.055798135697841644, 1.1742578744888306, -0.40913650393486023, 0.35233262181282043, 0.8335675597190857, 1.1972306966781616, 1.1309621334075928, 0.2923564612865448, 0.4428209066390991, -0.4437166154384613, 0.9726779460906982, 0.9418317079544067, -1.507829189300537, -0.3265627920627594, -0.41935956478118896, -0.9244285225868225, -1.4957329034805298, 2.153536319732666, 0.461045503616333, -0.21700799465179443, 0.959023118019104, -1.246728539466858, 1.4908002614974976, 0.09887795150279999, 0.49013832211494446, 2.1856203079223633, 0.6177099347114563, 0.5123371481895447, 1.4646339416503906, 0.5690948367118835, 0.3746304512023926, 0.6238932013511658, 0.8118091225624084, -0.6511749029159546, -0.20786906778812408, -1.410948634147644, 2.716548204421997, 0.1043391153216362, 0.9766587615013123, 1.1720943450927734, 0.2878482937812805, 0.1623144894838333, 0.8896286487579346, 1.2870372533798218, 0.7072840332984924, -1.4285494089126587, -1.1372382640838623, -0.41730037331581116, -1.1450393199920654, -0.955345869064331, 0.0604412816464901, 1.5134567022323608, -0.9642703533172607, 0.9353642463684082, 1.561344027519226, -0.9865761399269104, -1.7793726921081543, -0.5900636315345764, -0.10768716037273407, -0.5519475936889648, 2.7089149951934814, 1.149448275566101, -0.3682837188243866, 0.16232390701770782, -0.06857611238956451, 0.5037473440170288, 0.549109697341919, -0.38682931661605835, -0.15217001736164093, 0.5205080509185791, 0.5674306154251099, 0.154353067278862, -0.30239027738571167, -1.4826418161392212, 0.331156462430954, 1.3251274824142456, -1.1507551670074463, 0.16256996989250183, 0.7318640947341919, -0.7246087193489075, -0.21325252950191498, 1.38280189037323, -1.7467750310897827, -1.2822679281234741, 1.1245037317276, -0.5801570415496826, 0.21314242482185364, -0.24563488364219666, 1.5242797136306763, 0.6550229787826538, 0.6478287577629089, -1.741612195968628, -0.6907572150230408, -1.2678892612457275, 0.2670309841632843, -0.16278140246868134, 0.9375967383384705, 1.7409223318099976, 0.12385103851556778, -1.5132519006729126, -0.3524061143398285, -1.2960188388824463, -0.5254405736923218, -0.9485745429992676, 0.6562178134918213, 1.2689985036849976, 2.1488454341888428, -0.8608358502388, 0.4766097068786621, -0.1457933634519577, -0.11618445068597794, 0.3531971275806427, 0.31476321816444397, -0.06186951324343681, -1.2481392621994019, 0.11895264685153961, 0.991638720035553, -1.5470678806304932, -0.2593587338924408, -0.08274264633655548, -0.011614929884672165, -1.9772297143936157, -0.9556950926780701, -0.30237817764282227, 0.17928241193294525, 0.9109020829200745, -0.28229159116744995, 2.3704304695129395, -0.23428954184055328, 0.16817067563533783, -0.9014647006988525, -1.87765371799469, 0.17540805041790009, -0.30940061807632446, 0.6486572623252869, 1.5231220722198486, -1.3893989324569702, -0.3759085237979889, 1.8917216062545776, -1.3046743869781494, -0.2318158596754074, -1.0209200382232666, -0.37882208824157715, -0.5564782023429871, -0.8853451609611511, 0.3484554886817932, 0.31693798303604126, 1.5024269819259644, 0.19673921167850494, -0.6806079149246216, -0.2034575492143631, 0.022364730015397072, 0.36469873785972595, -0.19112655520439148, 0.31009310483932495, -0.25986340641975403, -1.732587456703186, 1.6957885026931763, 0.5420598387718201, -0.04998421296477318, -0.6103577613830566, -0.43797385692596436, -1.1867432594299316, 0.6949588656425476, -0.2164790779352188, 1.1898826360702515, -0.04305461421608925, 0.9033225774765015, -1.9412884712219238, 1.4941853284835815, -0.11428128927946091, -0.07274802774190903, -0.07771468907594681, 1.254800796508789, -0.1090928241610527, -0.6274904012680054, 0.49108731746673584, -0.21425317227840424, 0.1298828125, -0.7904754877090454, -0.32091808319091797, -2.1958389282226562, 1.332451343536377, 2.4328527450561523, -1.646268367767334, -1.7388231754302979, 0.2066977471113205, -1.0634087324142456, -2.4762802124023438, 0.7103150486946106, 0.2170013189315796, 0.26940807700157166, 0.059948645532131195, -0.8989642262458801, 1.028709888458252, 1.4625438451766968, 0.6379518508911133, -0.6085050106048584, -0.21469222009181976, -0.6137602925300598, -2.411936044692993, 1.3451142311096191, 0.06935082376003265, 0.6614176034927368, -0.6620371341705322, 0.49707847833633423, -1.9856605529785156, -0.7843802571296692, 0.06350665539503098, -1.073390007019043, 0.7469596266746521, -1.2745749950408936, 0.8850041031837463, -1.6868151426315308, -0.11336509883403778, 2.1012401580810547, -1.5716243982315063, 2.507742166519165, -0.7990829348564148, -0.7480708360671997, -1.118247151374817, 0.5851805210113525, 0.07473816722631454, 0.6261142492294312, 1.5152876377105713, 0.2054114043712616, -0.6567414999008179, 1.7834311723709106, -0.5202792286872864, -0.6004246473312378, -0.9467775225639343, -1.3033899068832397, 0.5196115374565125, -0.7438549399375916, -0.03829415887594223, 0.6903786659240723, -0.7705328464508057, -1.886505126953125, -1.7035884857177734, 0.37869083881378174, -0.4113326668739319, -0.902394711971283, 0.03707626834511757, -1.0782124996185303, -1.1883834600448608, -1.2341803312301636, 0.7149109840393066, -1.1074259281158447, 0.782196581363678, -0.2583332359790802, -0.15247809886932373, 0.5334927439689636, -0.04721187800168991, 1.208644986152649, 0.6394932866096497, -0.2682611048221588, -0.8854655027389526, -1.2653335332870483, 0.06144346669316292, -0.37057003378868103, 0.42221373319625854, 1.5396746397018433, 0.3409368097782135, -0.829433262348175, -0.31968364119529724, -1.2247411012649536, 0.08329054713249207, 0.4798199236392975, 0.2454657405614853, 2.003185272216797, 0.269862562417984, 0.0798032358288765, -0.7851396799087524, -0.23801261186599731, 0.15474660694599152, -0.3378334641456604, 0.41798049211502075, -0.826634407043457, 0.2130054533481598, 1.0149824619293213, -1.0692226886749268, 0.7257395386695862, -0.2832808196544647, 0.08989894390106201, -0.05494590103626251, -0.8452820181846619, 2.011202812194824, 0.03363090008497238, -1.725788950920105, -0.08741053938865662, -0.07693097740411758, -0.5841692686080933, -0.12149148434400558, -1.1493417024612427, -0.27746230363845825, 0.6806211471557617, 0.2031124085187912, -1.5912187099456787, 0.34387698769569397, -1.1585179567337036, -0.9131370782852173, 0.4067046642303467, -1.151387095451355, 0.7705861330032349, 1.3627214431762695, 0.3107657730579376, -0.12616093456745148, 0.06486207246780396, 0.9641657471656799, 0.26383793354034424, -0.9584162831306458, 1.7370944023132324, 0.38255736231803894, 1.6518293619155884, -1.1647928953170776, 1.9699158668518066], [-0.4011983871459961, -0.407505065202713, -0.2880294620990753, -1.500734806060791, 0.6859127879142761, -0.9539610743522644, -0.8830544948577881, -0.558688223361969, 0.5875310897827148, 1.5997235774993896, 0.6204987168312073, 1.362622618675232, -0.9183588624000549, 0.23409093916416168, 0.9320156574249268, 0.4482089579105377, 1.9861876964569092, 0.6971621513366699, 0.46304619312286377, -0.7871183753013611, -1.0143064260482788, 0.6958162784576416, -1.1270549297332764, -0.7243101596832275, 0.4419861435890198, -1.4988996982574463, 0.16372589766979218, 0.49228304624557495, -2.572906970977783, -2.138465166091919, -0.8956063985824585, -1.1755326986312866, 1.9506515264511108, -0.4718714654445648, -0.3084593713283539, -0.4584219753742218, 0.34102800488471985, 0.830009400844574, 0.83404940366745, -0.22477935254573822, -0.4536648690700531, 0.16226989030838013, -1.8821766376495361, 0.19433528184890747, -1.33939790725708, -1.075569987297058, -0.24752894043922424, 1.3538150787353516, -0.40852823853492737, -0.878223180770874, 0.5878349542617798, 1.4081474542617798, -0.1611398309469223, -1.1157552003860474, -0.46065375208854675, 0.22460374236106873, 0.3032931387424469, -0.32021474838256836, 1.1085247993469238, 0.35815882682800293, -0.36337989568710327, -1.278787612915039, 0.07133325934410095, 1.010807991027832, 0.005383324343711138, -0.30270394682884216, -0.06146802753210068, 0.7037961483001709, 0.2351832240819931, 0.6153003573417664, -1.6940116882324219, 0.9798569083213806, 0.4836527109146118, 0.4375775456428528, 1.0172817707061768, -0.4832291007041931, -1.298713207244873, 0.6971559524536133, 1.5575292110443115, -0.18180358409881592, 0.6140583753585815, 0.4913322329521179, 0.5344931483268738, 1.041502594947815, -1.7740330696105957, 0.9091698527336121, 0.32669273018836975, -1.307387113571167, -0.7882429957389832, -0.3837042450904846, 2.1167142391204834, -0.921420156955719, 1.7290465831756592, -0.5373650193214417, -1.4906302690505981, -0.08909976482391357, -0.5543833374977112, 0.8120102882385254, -2.16703200340271, 0.02848096564412117, -0.25022247433662415, 0.03745921701192856, 1.1953552961349487, -1.055679202079773, 0.5400681495666504, -0.1783703714609146, -0.6245324611663818, -1.5375758409500122, 0.32615047693252563, -0.47638800740242004, 0.542969822883606, -0.5027992129325867, 0.6420201063156128, -0.5251728296279907, -0.7140297889709473, 0.6010599732398987, -0.003916870336979628, -0.9434818029403687, -1.9365177154541016, -0.5260018706321716, -0.8754071593284607, 0.6956691741943359, 0.018346766009926796, -0.19767017662525177, -0.5107234120368958, -0.07170642912387848, -1.4867966175079346, 0.29393190145492554, -2.2508161067962646, -2.336707353591919, 0.2769234776496887, -1.434863805770874, -0.5024291276931763, -1.5128579139709473, 0.025507455691695213, 1.1136410236358643, 0.9455627202987671, 0.03080013021826744, 0.6980863809585571, 0.3850402235984802, 0.7485095262527466, 1.5034657716751099, -1.156387448310852, -2.092818260192871, 1.1468137502670288, -0.4371787905693054, 0.04102691635489464, 1.2018542289733887, -0.34436139464378357, -0.708635687828064, 0.4667607843875885, -0.21505038440227509, 1.700594425201416, -0.6869025826454163, -0.48312366008758545, -1.6640572547912598, 1.2361352443695068, 1.2974755764007568, -0.39636605978012085, -1.0349324941635132, 2.4322540760040283, -0.6685134172439575, 0.6316207051277161, -1.7329277992248535, -0.5790081024169922, -1.1167181730270386, 0.7019951343536377, 1.8216216564178467, 0.5802766680717468, 0.20009243488311768, 0.3497822880744934, -0.11803236603736877, 0.7607951760292053, 0.31924304366111755, -0.5863126516342163, 0.43339839577674866, -0.3346472978591919, -0.9838848114013672, 0.6578232645988464, 1.089870572090149, -0.9419046640396118, 0.19348877668380737, -0.9273871779441833, 0.003955117426812649, 0.7064962983131409, 0.9772051572799683, 0.1772400289773941, -0.5630417466163635, 0.18637549877166748, 0.2773345112800598, 0.4804787337779999, 1.802534818649292, 1.0976743698120117, 0.49258869886398315, -0.2516627609729767, 0.19603592157363892, 1.868706226348877, 2.2387444972991943, 2.1019420623779297, -1.4119634628295898, -1.7137497663497925, -1.0101763010025024, -0.5679805874824524, -1.0680917501449585, -0.24417619407176971, -0.619257390499115, 0.8718382716178894, -2.0233116149902344, 0.5514912605285645, -0.8494464159011841, -0.7719064354896545, -0.3134399950504303, -2.282991647720337, 0.24958524107933044, -0.37026670575141907, -0.9162437319755554, -0.34584271907806396, -0.32450053095817566, 0.4201604425907135, -0.3299243450164795, 1.0837148427963257, 1.0566576719284058, 1.4019653797149658, -0.12754935026168823, -0.1692453771829605, -0.6733272075653076, 0.27793624997138977, 0.5670812129974365, 0.15314769744873047, -0.10158969461917877, -1.4696742296218872, 0.7692402601242065, -2.012380599975586, 0.27221545577049255, -1.405167579650879, -1.0102225542068481, 0.6324508786201477, -0.9632897973060608, -1.3161163330078125, 0.19459064304828644, 1.0084891319274902, -0.23153744637966156, 0.4232366681098938, 1.2597684860229492, -0.26311665773391724, -1.408743143081665, 0.11218486726284027, 0.20487941801548004, -0.30723512172698975, 0.4922138452529907, 0.27450448274612427, 1.6244323253631592, 0.6040620803833008, 0.09037762880325317, 0.2749415636062622, -0.056560151278972626, 0.48645058274269104, -0.23742741346359253, -2.0032808780670166, 2.0302631855010986, 0.11999097466468811, 1.31101655960083, -0.4784198999404907, -1.0171254873275757, 0.9460597634315491, 0.009358279407024384, -1.022468090057373, -0.9179989695549011, 0.43155500292778015, 0.3310680389404297, 2.246696710586548, 0.8650619983673096, 0.7713078260421753, -0.8351035714149475, 0.19045324623584747, -0.29910749197006226, 0.014101712964475155, -0.7772263884544373, -0.855989396572113, -1.4521853923797607, -1.903852105140686, -1.1983855962753296, 0.008995812386274338, 0.16873697936534882, -1.6593878269195557, -0.33374956250190735, 1.1202384233474731, 0.6272332668304443, -2.1459872722625732, -0.5412264466285706, -0.4249266982078552, -0.8866307139396667, -0.17673718929290771, -1.2112658023834229, 0.23873348534107208, 1.8103059530258179, 1.6587845087051392, 0.8150413632392883, -0.8774154782295227, -0.9704543948173523, -1.069245457649231, -0.44068220257759094, -0.6355558633804321, 0.5296431183815002, -1.3419662714004517, -1.5663013458251953, -0.4053414762020111, -0.8560205101966858, 0.8300600647926331, -1.7744965553283691, 0.3901282250881195, -0.4105607569217682, 0.23844879865646362, 0.2863498032093048, 0.6246697306632996, 1.3150635957717896, 1.0372159481048584, 0.5203438401222229, 0.31939074397087097, 0.16989754140377045, -1.4593254327774048, 0.6770074367523193, 1.125613808631897, -0.052170924842357635, -0.46356579661369324, 1.1574687957763672, 2.855316162109375, -1.1110336780548096, 0.3925436735153198, 0.09672313928604126, -1.307181477546692, -0.807682454586029, 0.8572301864624023, 0.23467247188091278, 0.02385806292295456, -1.1968134641647339, 0.449497252702713, -0.9486063718795776, 1.4182404279708862, -0.7530711889266968, -0.6340038776397705, 0.9149759411811829, 0.07065809518098831, -0.8101586699485779, 0.8623098134994507, -0.17205817997455597, 0.5571738481521606, 0.7957385778427124, -1.0757046937942505, 2.059605598449707, -0.706900417804718, -0.798529863357544, 0.07009167969226837, -0.32533201575279236, 0.8456013202667236, 0.8971567153930664, 0.2943781912326813, -0.37465646862983704, 0.3616403639316559, 0.1649191975593567, -0.6569943428039551, -0.28706157207489014, -1.5927826166152954, -0.7055641412734985, -1.0634316205978394, -0.5441079139709473, 0.9880167841911316, 1.501259684562683, 1.4181041717529297, 0.6763426661491394, 1.0568723678588867, 0.16991101205348969, 0.716097354888916, -1.1650161743164062, 0.4063652455806732, 1.8905236721038818, -0.23931054770946503, 0.8875024318695068, 0.2651577591896057, -0.6806410551071167, 0.4795401692390442, 0.8609009385108948, -0.575456440448761, -0.16170278191566467, -0.18624545633792877, -1.4742090702056885, 0.6008232235908508, 0.3732430636882782, 0.0413559190928936, -0.2391970306634903, -0.3351764678955078, -0.3504472076892853, -0.49714231491088867, -0.6246985197067261, 0.19944708049297333, 1.1485943794250488, 0.25915080308914185, -0.03800968453288078, 1.9761818647384644, 1.5544102191925049, -0.9280356764793396, -1.4728355407714844, -0.738230288028717, 0.36211976408958435, -0.9858604669570923, -0.6287989616394043, -0.2572871744632721, 0.2332126945257187, 0.06082506850361824, -0.8705297112464905, -0.6867173314094543, -1.5179309844970703, 0.3404681980609894, -0.4462517201900482, -1.4243505001068115, 1.1718494892120361, -1.5195367336273193, 0.1584722101688385, 0.29303234815597534, 0.43186765909194946, 0.8410773277282715, -1.3161345720291138, -0.2569951117038727, 0.2807922065258026, -1.3526420593261719, 0.09532652050256729, -1.5725436210632324, 0.4024699628353119, 1.0937833786010742, -0.05496073141694069, -0.042774610221385956, -0.4674202799797058, 1.5963839292526245, 0.19917504489421844, -0.15061773359775543, 0.40476784110069275, 1.309264063835144, -0.7665362358093262, -0.5390406250953674, 0.6597203612327576, -0.2702755033969879, 0.16411221027374268, -0.7810510396957397, 1.1464512348175049, -0.2018776535987854, -1.364301085472107, -0.9622459411621094, -0.518644392490387, 1.5388596057891846, 1.1754342317581177, -0.12401064485311508, 1.2418142557144165, -1.9026786088943481, 0.6516050696372986, 0.4993838965892792, -2.710336446762085, 0.8707401752471924, -1.4120774269104004, -1.002109408378601, -2.826014995574951, 1.1522473096847534, -0.3986823856830597, -0.4273582994937897, 0.6850958466529846, -0.9245379567146301, -2.1646015644073486, -0.7793091535568237, 1.8714089393615723, -1.1222285032272339, 1.0895782709121704, 0.9595078229904175, 0.7100717425346375, 0.2995985746383667, 1.5392733812332153, 0.6002699732780457, 1.2207672595977783, 0.11371737718582153, -0.7286349534988403, 0.5468544363975525, 1.1416395902633667, -0.2694465219974518, -1.620499610900879, -1.8616526126861572, 1.664393424987793, 0.08177616447210312, -0.7007917761802673, -1.2891864776611328, -1.4688931703567505, -2.3178560733795166, -0.9954843521118164, 0.13109305500984192, 0.7397642135620117, 1.5583475828170776, -0.2652115225791931, 0.20960365235805511, 0.0694398432970047, -1.4502782821655273, 0.31703224778175354, -0.16705608367919922, -0.2516176402568817, -0.08405513316392899, -1.463363766670227, -0.2509341537952423, -0.8207200169563293, -0.3713681399822235, -0.8831505179405212, 1.0895556211471558, -1.2391337156295776, -0.8708818554878235, 1.1333309412002563, 0.1916079819202423, -0.6455956697463989, -0.7897283434867859, -1.2505125999450684, -0.4926826059818268, 0.16627579927444458, -0.1028481051325798, 0.16843625903129578, 0.5177373886108398, 1.1103718280792236, 0.4039177894592285, 1.5377466678619385, 0.16832193732261658, 0.30296850204467773, -1.0473474264144897, 0.9782576560974121, 0.39120519161224365, -0.258125364780426, -0.5515960454940796, 0.23515987396240234, -0.9087232351303101, -0.31735819578170776, -1.557528018951416, 0.6176707148551941, 0.5868896842002869, 1.7281752824783325, 0.7449439167976379, -0.21931985020637512, -0.409000962972641, 1.9473528861999512, 0.3766472637653351, -0.6718521118164062, 0.529066801071167, 1.8945237398147583, -0.9577540159225464, -0.29554691910743713, -2.24672269821167, 1.1752383708953857, -0.5213841795921326, 0.015576334670186043, 3.164928913116455, -0.8137210011482239, 1.2043688297271729, 1.3113484382629395, 1.3286521434783936, -1.0255793333053589, -0.2830975651741028, 1.0179991722106934, -0.7544952630996704, -2.1444427967071533, 0.5280467867851257, -1.425156593322754, -1.1234415769577026, 1.3865232467651367, 1.293715238571167, 0.41419968008995056, -0.551752507686615, -0.3366103768348694, -0.22126825153827667, 0.6343482136726379, 0.6770788431167603, -1.1273255348205566, -0.42109066247940063, 0.014259273186326027, -0.7080546021461487, 0.02029659040272236, 0.607853353023529, -0.26994988322257996, -0.08839109539985657, 0.8371796011924744, -1.495713233947754, -2.1238155364990234, 0.5563685297966003, 0.4023630917072296, 0.022325625643134117, 0.10713787376880646, 0.6051136255264282, -0.3719714283943176, 0.21649229526519775, -0.2236822247505188, 1.5429484844207764, -0.43280237913131714, -0.9575656652450562, 0.3374880850315094, -1.3584692478179932, 0.6761568188667297, -1.3039134740829468, -0.6231495141983032, 0.8591710925102234, -0.17686043679714203, -0.12420438230037689, 0.2726123034954071, -0.6699920296669006, 1.6082285642623901, -0.027086451649665833, -0.6342247128486633, -0.00046107597881928086, -1.7996503114700317, -1.0113524198532104, -0.10650226473808289, -0.559641420841217, 1.1747015714645386, 0.10422395169734955, 0.7679891586303711, -0.6180565357208252, -1.627234697341919, 0.43835097551345825, -2.5501840114593506, -1.087815761566162, -0.8986336588859558, 1.573232650756836, 2.0643362998962402, -1.5541515350341797, -0.7088988423347473, 1.3491977453231812, 0.7539824843406677, -0.1429385095834732, -0.10033212602138519, 0.39467328786849976, -0.5166242122650146, 0.042915698140859604, 0.491595983505249, 0.14585258066654205, -0.1212366595864296, 0.15257930755615234, 1.4283804893493652, -0.156996488571167, -0.37720829248428345, 0.08580297976732254, -0.16307464241981506, 0.3808181583881378, 0.4093446731567383, -0.16315023601055145, 0.16809768974781036, 0.024349123239517212, -1.5127211809158325, 0.13746632635593414, -0.05189927667379379, 0.6975414752960205, -1.1559810638427734, 0.7303124070167542, -0.7171351909637451, -1.499727725982666, 1.124168872833252, -0.2951671779155731, -1.1545222997665405, -0.8410781025886536, -0.3930096924304962, 1.0085182189941406, 0.3332310914993286, -0.8194318413734436, -0.9571793675422668, -0.5659137964248657, 0.2726570963859558, 0.786894679069519, -1.1022288799285889, -0.8250101208686829, -0.42143356800079346, -0.8462666869163513, 0.3148047924041748, -0.5275078415870667, 0.04568943753838539, 0.9662623405456543, -1.9387930631637573, 0.5867295265197754, 0.79322749376297, 0.11618078500032425, -1.210287094116211, 0.6948212385177612, -0.26693469285964966, 1.2593026161193848, 0.16446281969547272, -0.6985983848571777, -1.1556645631790161, 0.7528188228607178, 0.09260443598031998, -1.6070683002471924, 0.17457756400108337, 1.1435158252716064, 2.3467864990234375, 0.6250866651535034, 1.015824794769287, -0.05880255624651909, -0.9612374305725098, 0.6924300193786621, 1.4018226861953735, -2.1981759071350098, 1.314331293106079, -2.4223947525024414, 0.3776526153087616, 0.0874493345618248, 1.8976534605026245, -0.700835108757019, 0.03374901041388512, -1.4405431747436523, 0.6471297144889832, -0.12873686850070953, -0.20682759582996368, -0.7291389107704163, -0.05009016394615173, -0.3877127766609192, -1.7070128917694092, -0.6185232400894165, -0.06554920226335526, -1.0289487838745117, -1.0828748941421509, -0.15663085877895355, -1.2611057758331299, 0.885231077671051, -0.5493894815444946, -1.8170353174209595, -0.7413914203643799, 0.3809247314929962, -1.6142581701278687, -1.3926217555999756, 0.38045522570610046, -0.3311537206172943, -0.390912801027298, 0.6578337550163269, -0.5438624024391174, -0.741533637046814, 2.204550266265869, -0.8128111362457275, 0.842939019203186, 0.2531471252441406, -1.3200933933258057, 0.21981284022331238, 0.6893951892852783, 0.7280961871147156, -0.8202190399169922, 0.6080594658851624, 0.3465425372123718, -0.7522836923599243, 0.4036839008331299, -0.33596450090408325, 0.4067919850349426, -0.2546694278717041, -0.600891649723053, -1.1490120887756348, 1.3225547075271606, -1.2322978973388672, 0.11664881557226181, 0.06613267213106155, 0.3437304198741913, -1.5680816173553467, -0.24300836026668549, -0.09109637141227722, 0.4492795169353485, 0.18537651002407074, -0.16513341665267944, -0.0031057740561664104, 1.4162869453430176, -0.48055499792099, -0.03496687114238739, -0.16195985674858093, 0.9233314990997314, -0.6582023501396179, 0.47863656282424927, -0.8503865003585815, -0.47009438276290894, -1.1706056594848633, 2.582581043243408, -0.07970131933689117, -1.557336449623108, -1.5925759077072144, -0.7796568274497986, -1.4619810581207275, 0.563701331615448, -1.499407172203064, -0.6136842966079712, -0.7405621409416199, -1.822252869606018, -1.1072723865509033, -1.3310692310333252, 1.648759365081787, -0.845988929271698, -0.22472189366817474, -1.9718235731124878, -0.8861774802207947, 0.4864124059677124, -0.49309617280960083, 0.9407455325126648, -1.00752854347229, 1.9691811800003052, 0.9099193811416626, -1.821002721786499, 1.2393587827682495, -1.7270375490188599, -0.42476174235343933, 2.1326537132263184, 1.8419349193572998, 0.4818754196166992, 0.9173017144203186, -0.1332743912935257, 0.4753415882587433, 1.2343640327453613, -1.1369600296020508, -1.9341365098953247, 0.9308416843414307, 0.5657857656478882, -0.27022784948349, -1.513687014579773, -0.21513302624225616, -0.5311441421508789, 0.44794389605522156, 0.7290863990783691, 1.5359920263290405, -1.2311420440673828, 0.9404414296150208, 0.22798508405685425, -0.6852381229400635, 0.07519228011369705, -1.432331919670105, -0.49332550168037415, 0.7160026431083679, -1.9781168699264526, -0.5512843728065491, -0.17290227115154266, 0.8612649440765381, 1.8765220642089844, 1.16981041431427, -0.9495157599449158, -1.148880124092102, 0.6867609620094299, 0.902390718460083, 1.6706212759017944, -1.277945637702942, 0.11111648380756378, -1.1740005016326904, 0.8653885722160339, -1.0990328788757324, 1.3076788187026978, -0.07398445904254913, 0.17230187356472015, -1.7171096801757812, -2.652796983718872, 0.43415015935897827, -1.5831151008605957, 0.16803547739982605, -0.9521135687828064, -0.19348961114883423, 0.6272262334823608, -2.2572362422943115, 1.2371858358383179, -0.15629565715789795, -0.4145752191543579, 0.2661552131175995, -0.531741201877594, -0.0666208565235138, -0.11523233354091644, 0.1285666972398758, 0.3317960500717163, 0.41266849637031555, 0.7868954539299011, 0.8614561557769775, -0.6558979153633118, 0.004803135059773922, -0.9077988266944885, -1.6667206287384033, -2.5429232120513916, -0.555962860584259, 1.7225652933120728, -0.36169058084487915, 0.9634559750556946, 0.7215173244476318, 1.3944671154022217, 1.8412240743637085, -0.1951870620250702, -0.46331262588500977, -0.13440509140491486, 2.1963162422180176, 0.7361781001091003, 1.2709934711456299, -0.2072204202413559, 0.3926045894622803, 1.2433838844299316, -2.1523611545562744, -0.4056139886379242, -0.7210004329681396, 1.0552356243133545, 0.7029061913490295, 2.1826066970825195, -2.1989121437072754, -1.297924518585205, 1.0325196981430054, 0.4504398703575134, -1.0987765789031982, 0.9436230063438416, 0.42070654034614563, -0.7384406924247742, -1.3647183179855347, -0.8325554728507996, 0.0364551916718483, 1.2339370250701904, -0.07426279783248901, -1.5815860033035278, -1.1443355083465576, 1.161670446395874, -0.15195828676223755, -0.6227642893791199, 1.7767552137374878, 0.06602317094802856, -0.008019699715077877, -0.07794131338596344, -1.8070216178894043, 1.4765552282333374, 0.47744622826576233, 1.056750774383545, 1.3235856294631958, -1.1416652202606201, -0.9706057906150818, 0.5773839354515076, -0.7354419827461243, -0.8208790421485901, -1.6435424089431763, 0.4605366289615631, 1.039811134338379, -0.6191499829292297, -0.7653723359107971, 2.558433771133423, -0.6304599046707153, 0.8748633861541748, 1.636755108833313, -1.2970088720321655, 0.9671460390090942, 2.112860679626465, -0.3407200276851654, -1.1590986251831055, -0.7540706992149353, 0.6319350600242615, 0.2669849693775177, -0.11283624172210693, -0.585305392742157, 0.5531694889068604, 2.5070385932922363, -1.4225763082504272, 0.09685271978378296, 0.1747433841228485, -0.003383104456588626, 0.28501489758491516, -0.2555379271507263, 0.7583989500999451, -1.280859351158142, -1.2479416131973267, -0.5582594871520996, 0.10975547879934311, -0.24274127185344696, 0.6694040298461914, 0.7986873984336853, 0.6824645400047302, 1.4816564321517944, -0.04397355765104294, 0.8938106298446655, -1.4217287302017212, -1.4773064851760864, 0.8338241577148438, -0.10629615187644958, -1.3383318185806274, 0.33684298396110535, 0.6124767661094666, 1.0616910457611084, 1.9707703590393066, -0.6755616068840027, -0.6497629880905151, 1.4343156814575195, -1.2402355670928955, 0.6400119066238403, -1.1105766296386719, -0.6011614799499512, 0.9671927094459534, 0.291259765625, 0.04031873121857643, 1.5000327825546265, -0.3863166570663452, 0.6137881278991699, -0.10541345179080963, -0.6479535102844238, 0.8096923828125, -0.06590104103088379, -0.4332960247993469, -0.5979227423667908, -1.970689058303833, 2.0840892791748047, 1.4168094396591187, -0.8391945958137512, 1.2397818565368652, 0.8669479489326477, -2.6936542987823486, -1.0292134284973145, 0.49970486760139465, -0.05034799873828888, 1.550856590270996, 1.8411709070205688, -0.8189055919647217, 1.5150904655456543, -0.8233091831207275, 0.16642770171165466, 0.403021901845932, -0.7659128308296204, -1.7628741264343262, -1.0710808038711548, 1.2422730922698975, -2.3625433444976807, 1.1446155309677124, 0.9662341475486755, 0.6925696730613708, 1.2028460502624512, 0.46638426184654236, -1.1582415103912354, 0.3471399247646332, 1.3573018312454224, 0.7599087953567505, 0.6612026691436768, -1.4560635089874268], [1.6496131420135498, 2.371396064758301, 1.353193998336792, 0.8146512508392334, -2.1310465335845947, 0.07396915555000305, -1.2951045036315918, -0.7991505265235901, 0.047155581414699554, -1.0588141679763794, -0.3360172212123871, 1.092774510383606, -0.014843014068901539, 1.8925604820251465, -0.13537712395191193, 0.39223387837409973, 1.5324172973632812, -1.287557601928711, -2.449763298034668, 0.5668929815292358, 0.2411777824163437, 0.9102053046226501, 0.9076358079910278, -0.5612245202064514, 0.2694331109523773, 0.20031002163887024, -1.6243889331817627, 0.1842430979013443, 1.8113535642623901, 0.0973813533782959, 0.7114009857177734, 1.1173691749572754, 0.20222283899784088, -0.5629341006278992, -0.2749055325984955, -0.6976725459098816, 0.3200337290763855, 0.4011729955673218, 0.36803972721099854, 0.46376317739486694, -0.7069153189659119, 0.1633998602628708, -0.3039984405040741, 0.10518477857112885, 0.0972251147031784, 0.21561576426029205, -1.8237332105636597, -0.09842804819345474, 0.6573537588119507, 0.3205089569091797, -1.8653146028518677, 1.523162603378296, 1.7938718795776367, -1.319570541381836, 0.2460031807422638, 0.9153207540512085, -0.2873344421386719, 1.1169267892837524, -0.21269728243350983, 0.14553521573543549, -0.803524374961853, 0.7155900001525879, -0.5404205322265625, -1.3877620697021484, -0.1980704814195633, 0.6726198196411133, -0.019250843673944473, 0.10726696252822876, 1.1578881740570068, 0.11499954760074615, -0.43867427110671997, -0.11601045727729797, -1.6982591152191162, -1.6138440370559692, -0.37308600544929504, 1.404310703277588, -0.12479253858327866, -1.8306485414505005, 1.0044901371002197, -0.09554319828748703, 0.060539424419403076, 0.531216025352478, -0.7675027251243591, -1.3106273412704468, 0.5129711627960205, -0.7876288890838623, 1.2889472246170044, 0.6127504706382751, -2.985517978668213, -0.11734456568956375, 0.3157352805137634, 0.23335784673690796, 0.4244791567325592, -0.9152378439903259, 0.2879868149757385, 0.5688074231147766, 1.3300162553787231, -0.3881532847881317, -0.6957007050514221, -1.0215294361114502, 0.28164541721343994, -0.09721734374761581, -0.30958467721939087, 1.6699767112731934, 1.3149871826171875, -0.5957084894180298, 0.05337364226579666, -0.7282699942588806, -0.07720617204904556, -1.1821857690811157, 0.5893394351005554, 1.6407980918884277, 0.4806821048259735, -1.1028193235397339, -0.423261821269989, -0.5968761444091797, -1.0250378847122192, -1.5952014923095703, 0.2515716254711151, 0.5806405544281006, 0.12355271726846695, 1.2053943872451782, 0.3315090239048004, -2.7960290908813477, 0.5928170084953308, -1.3719587326049805, -1.060880422592163, 1.335594892501831, 1.5391345024108887, 0.22308491170406342, -0.008779691532254219, 0.42549455165863037, 1.7088203430175781, 0.5949856042861938, 0.1714722216129303, -0.35997679829597473, 0.5805202722549438, 0.10688706487417221, -1.234737753868103, 0.015772655606269836, -0.5245574116706848, 0.9406079649925232, 0.6266711354255676, -0.3800784647464752, 0.8801414370536804, -0.48161780834198, -0.3422633111476898, -0.361225426197052, -0.8204396963119507, 0.4054473638534546, -0.38889673352241516, 0.5960094928741455, 1.2794830799102783, -0.17082633078098297, -1.0276708602905273, -0.48864486813545227, -0.07945938408374786, -1.02603018283844, 0.05435239151120186, 2.363921642303467, 1.2953320741653442, 0.05534939467906952, 1.3983371257781982, 0.4531260132789612, 0.44325926899909973, 0.4223339259624481, -0.9355388283729553, -0.08531155437231064, 1.4993785619735718, 1.6485790014266968, -0.31939592957496643, -0.13335858285427094, -0.7442700862884521, -0.3692922592163086, 1.270777940750122, 1.739638328552246, -1.0317851305007935, 0.008297811262309551, -0.8760122656822205, 1.996588110923767, -0.5387138724327087, 0.023367760702967644, -0.3058541417121887, 0.09070181846618652, -0.5181664824485779, 0.656025230884552, 0.08751724660396576, -0.8670547008514404, -0.9473370313644409, -1.1295543909072876, -0.10463032871484756, 0.07111715525388718, 0.8774012327194214, -0.5888351798057556, 0.35966625809669495, 1.5301936864852905, 0.8926624655723572, -0.06144598871469498, 0.6058923006057739, -1.1004847288131714, -2.0375776290893555, -1.1146525144577026, -0.4896654486656189, -0.3875174820423126, -1.8388842344284058, -0.9723241925239563, 1.6411446332931519, -1.0911635160446167, -0.18441498279571533, -0.6905554533004761, -0.28824344277381897, 0.014540477655827999, 1.569138765335083, -1.2648613452911377, 0.26673898100852966, 2.4614689350128174, -1.676709532737732, -0.3416317403316498, -0.29969295859336853, -1.2236273288726807, -0.6617369651794434, 0.011155025102198124, -0.17050503194332123, -0.05008469894528389, -1.4764840602874756, -0.12358178198337555, 1.0118072032928467, -0.1417577564716339, -0.10742355138063431, -2.185979127883911, -1.1660895347595215, -0.8991982340812683, 0.6002058982849121, -0.7553776502609253, -1.8852815628051758, -0.7104588747024536, 0.7252794504165649, -0.15478813648223877, -0.3394559621810913, -1.7894299030303955, 0.3622815012931824, -0.918743371963501, -0.7364075183868408, 0.8011842370033264, 0.33555829524993896, 0.2831583023071289, 1.7113233804702759, -0.4665284752845764, 0.2948261499404907, 0.1600874662399292, 0.7698203325271606, -0.32681581377983093, -1.250471591949463, 1.2013651132583618, -0.3818794786930084, -0.48218584060668945, 1.9875519275665283, 0.8951666355133057, 0.6076085567474365, -0.6325260996818542, -0.854682981967926, -0.059916771948337555, 0.9876700043678284, 0.4944700300693512, 1.2755839824676514, -0.8316506743431091, -0.15497153997421265, 1.2925077676773071, -0.3894580900669098, 0.6468979120254517, 2.1782636642456055, -0.7913762927055359, -1.848918080329895, -0.5810996890068054, -0.26286542415618896, -1.4783190488815308, 0.7958493232727051, 0.7352141737937927, 0.1257876753807068, 0.41488561034202576, -0.6695076823234558, -0.5261377692222595, -0.77245032787323, 0.5577914118766785, 0.9437357783317566, 0.30455008149147034, -1.0613352060317993, -2.5488603115081787, -0.6952491402626038, 0.480589896440506, 0.6475645303726196, -0.798096239566803, -0.978076696395874, 0.06953389942646027, 1.4565869569778442, 1.1580747365951538, 2.849869728088379, -0.03613952174782753, -0.6720414757728577, 0.8297654986381531, -1.0535118579864502, -1.2897217273712158, 1.1096521615982056, -0.21797165274620056, -0.8749955296516418, -0.07603315263986588, 0.33465591073036194, -1.1060487031936646, 0.8159573674201965, -1.3086906671524048, -0.45208677649497986, 0.9059558510780334, -0.6246657371520996, 0.22719351947307587, -1.0165022611618042, -0.8224168419837952, -1.6562678813934326, -0.22455045580863953, -0.3794780671596527, 1.3697494268417358, 0.7352273464202881, 0.8774328827857971, 0.6062123775482178, 0.42647621035575867, -0.31071531772613525, 1.8414250612258911, -0.6357381939888, -1.1083117723464966, -1.3055540323257446, 0.3158782422542572, -0.2646201252937317, 1.3762800693511963, -0.3174084722995758, -0.35966089367866516, 0.06730173528194427, 1.0843685865402222, 0.8731963634490967, 0.9151011109352112, 0.6031620502471924, 1.1487547159194946, -0.9047539830207825, 0.5501718521118164, 0.1858154982328415, 1.2678921222686768, 1.369053602218628, -1.9785212278366089, -0.11396560817956924, -0.6906967759132385, -0.4021399915218353, -0.782203197479248, 1.2261815071105957, -1.984840750694275, -0.3160923719406128, 1.641090750694275, -1.030866026878357, 1.405142903327942, 1.258893609046936, -0.13230502605438232, 0.6291501522064209, 0.9686791896820068, 2.215984344482422, 1.4861395359039307, -0.4361226260662079, -0.7175418138504028, 0.2473856657743454, 0.4256802499294281, 0.488310307264328, 0.2341095209121704, 0.48194172978401184, 0.08406661450862885, -0.39234453439712524, -0.14760330319404602, 1.710663914680481, 0.9379283785820007, 0.0666179433465004, 1.8618547916412354, 1.430085301399231, -2.173304557800293, 1.023857831954956, -0.19404053688049316, -0.4900471866130829, 0.0719396099448204, -1.3613090515136719, 0.22224199771881104, -0.9367278218269348, 1.619140863418579, -0.09033486247062683, -1.4525153636932373, 0.7319382429122925, 0.06342029571533203, 1.7694063186645508, 0.0722903311252594, 0.26230093836784363, 0.05475250631570816, 1.0699445009231567, 1.5587891340255737, 0.01821998693048954, 2.2497854232788086, 2.0128023624420166, -0.9773979187011719, 0.9423075914382935, -0.17016489803791046, 0.9017904996871948, 1.2130554914474487, 0.8514485359191895, -0.5496860146522522, -0.2717929780483246, -0.7294607758522034, -2.61374831199646, 0.6215165257453918, 0.9323453307151794, 0.8112386465072632, 0.35615837574005127, -0.1836138367652893, 0.6813252568244934, -0.3036908805370331, 0.0876048356294632, 1.810921549797058, -0.5636193156242371, 1.0185004472732544, 0.637575626373291, 0.35569527745246887, -0.6346132159233093, 0.8177279233932495, -0.45272311568260193, -1.6068053245544434, 1.5402815341949463, 0.42810413241386414, -0.027191124856472015, -0.8905143737792969, -0.1873602271080017, -0.9030675888061523, 1.0491547584533691, 1.4663140773773193, 1.9080809354782104, 1.1629104614257812, 0.5074365139007568, 0.1532873511314392, 0.11447180807590485, -0.5131729245185852, 0.01902843452990055, 1.5225449800491333, -2.727548122406006, 1.022109031677246, 0.580566942691803, -0.6260506510734558, 0.6204020380973816, -0.2751610577106476, -0.43708693981170654, -1.2177056074142456, 0.8333854675292969, -0.12966665625572205, -0.8576751947402954, 0.7329748868942261, 2.22744083404541, 1.1713898181915283, 1.8015103340148926, -0.41047099232673645, -0.2109365463256836, -0.28275066614151, -0.9433256387710571, -1.3993788957595825, -0.6313745975494385, -0.7394250631332397, -0.5956109762191772, 0.5016208291053772, -2.1409196853637695, -0.16402406990528107, -0.753304123878479, -0.4215831160545349, -1.4847103357315063, -0.2680143713951111, 0.21134668588638306, 0.5353606939315796, 0.05308253690600395, -1.7500449419021606, -0.9294048547744751, -0.7639721035957336, 0.3273445963859558, -0.29892072081565857, 1.1949138641357422, 0.9130739569664001, 0.21924473345279694, -0.08746710419654846, -0.6681526899337769, 0.6572457551956177, 1.691097617149353, 0.26792073249816895, -0.12135691940784454, -1.239133596420288, 3.267690896987915, -0.7650228142738342, 1.9056216478347778, 1.024747371673584, -0.1466691941022873, -0.09523505717515945, 0.05904839187860489, -0.5990291237831116, 0.23143184185028076, 0.650230884552002, 0.3149871528148651, -0.7334898114204407, -0.19985337555408478, 1.0903308391571045, -1.756301999092102, -0.07229011505842209, -0.8969488143920898, -0.041366901248693466, -0.2555346190929413, -0.7060670852661133, 1.157118558883667, 0.4758468270301819, 0.5706351399421692, 1.1926312446594238, 1.559840202331543, 0.08396018296480179, -0.6784746050834656, 0.1986956149339676, -0.8093896508216858, -0.09980884939432144, 0.10851787775754929, 2.436962366104126, -0.36261796951293945, -0.8088089227676392, -0.32795700430870056, -1.5328015089035034, -1.8203679323196411, -0.19303631782531738, 1.3306313753128052, 1.1434131860733032, 0.916918158531189, 0.7076581716537476, 0.8526540994644165, 0.6170188784599304, 0.35749951004981995, 2.0059964656829834, -0.28607839345932007, -1.459240436553955, -0.14036071300506592, 0.25732943415641785, 1.2471204996109009, -1.2284870147705078, 1.3652929067611694, -1.124106764793396, 1.2988722324371338, -0.6785905361175537, 0.544815182685852, 1.5254765748977661, -0.7423840165138245, -0.5155681371688843, 0.7646827101707458, 1.650184154510498, 0.19433851540088654, -0.5646363496780396, -0.11948312073945999, -2.474787950515747, 0.10095701366662979, 0.001632438856177032, -0.5549261569976807, -1.307247281074524, -0.46571606397628784, 0.44384896755218506, 2.425447702407837, -0.7855224013328552, -2.0051846504211426, 0.07564076781272888, 0.7843337655067444, 0.08123829215765, 0.4992513954639435, 1.498367190361023, 0.0005076902452856302, 0.6717056632041931, 0.1561434268951416, 0.43071621656417847, 0.03405157849192619, 1.7809628248214722, 2.5108449459075928, -1.4353529214859009, -0.7517114877700806, 0.8683149814605713, 1.3599720001220703, -0.47662001848220825, -1.4773789644241333, 1.1820138692855835, 0.25547707080841064, 0.29639700055122375, -0.6474674344062805, 1.4208966493606567, -0.4083195626735687, -0.2179597169160843, 1.3983575105667114, 2.2575666904449463, -0.5900185108184814, 0.1389322727918625, -0.9270203113555908, 1.1021031141281128, 1.1836655139923096, 0.9456464052200317, 1.595994472503662, -1.8614780902862549, 0.559377908706665, -0.8263986110687256, -0.07987075299024582, -0.17055362462997437, 0.7534633874893188, 0.841766357421875, 1.025912880897522, 0.7366238236427307, -0.06973667442798615, 0.7448388934135437, 0.11069013178348541, 0.5613477230072021, -0.7698139548301697, -0.4220876395702362, -0.5773435235023499, 1.0270410776138306, -1.1255333423614502, -0.3296075761318207, 1.0073660612106323, 0.2347763478755951, 0.2903306782245636, 0.3019198477268219, -0.09345822781324387, 0.8906298279762268, 1.2187834978103638, 1.9971064329147339, 0.5259225964546204, 0.538403332233429, -1.0844416618347168, -1.8106977939605713, 0.5045056939125061, -0.05488625913858414, -0.5494431853294373, 0.9007496237754822, -0.4698333442211151, -0.4736970067024231, 0.9857126474380493, -0.10769526660442352, -0.17630107700824738, -0.5255717039108276, 0.6960800290107727, -0.16591034829616547, 0.611851155757904, -0.7161543369293213, 0.6345863938331604, 0.35029542446136475, 0.5666348338127136, 0.32328662276268005, 0.41164684295654297, 1.0691872835159302, 0.7191335558891296, 1.4333691596984863, 1.9774324893951416, -1.1646925210952759, 0.9955008029937744, 0.5440585613250732, 1.2267208099365234, -0.6484431028366089, 0.8405151963233948, -0.6947513222694397, 1.7393481731414795, 0.5052427053451538, 1.1235074996948242, 0.5338212847709656, 1.1485004425048828, -0.7090023756027222, -0.059340205043554306, -1.0274195671081543, -1.0738341808319092, 0.3480531871318817, 1.5837767124176025, 0.8689191341400146, 0.4393780827522278, -0.2569923400878906, -0.3608730733394623, -0.5980377197265625, 0.8509716987609863, -0.14964383840560913, -1.6893460750579834, 1.250357985496521, 0.8482623100280762, 0.02066592499613762, -1.150464653968811, 1.1137018203735352, -0.48249563574790955, -1.1520817279815674, 0.8398023843765259, -1.1182199716567993, -1.0003595352172852, -0.5498751997947693, -0.3412066400051117, 0.35553064942359924, -0.13707420229911804, 1.4559133052825928, -0.9479968547821045, 0.3213195502758026, -1.6532758474349976, -0.24026139080524445, 0.42245668172836304, 0.04616855829954147, 2.503556966781616, -0.3290833532810211, -1.9137508869171143, 2.23795747756958, -1.2703728675842285, 0.4236031770706177, 1.4438210725784302, 0.7462764978408813, 0.38413378596305847, 1.758398175239563, -0.19982552528381348, 0.18748345971107483, 1.5329644680023193, 0.6414562463760376, -1.3206133842468262, -2.121133804321289, 1.2454886436462402, 0.9713022708892822, 0.12125515937805176, -1.71713387966156, 0.41308560967445374, -0.5448793172836304, 1.4795809984207153, 0.06157705560326576, 1.5624582767486572, -1.3051176071166992, 1.2689406871795654, 1.165205955505371, 1.2713433504104614, -0.029324796050786972, -0.10378250479698181, -0.10228569805622101, -0.411570280790329, 0.9332334995269775, -0.125239297747612, -1.1403405666351318, 0.5784794688224792, -0.261422723531723, -0.40565013885498047, -0.6739143133163452, -1.5985742807388306, 1.5651038885116577, -0.8879404664039612, 1.0609086751937866, -0.03985117748379707, 0.8388347625732422, -0.1700150966644287, 1.6363189220428467, -0.6040880680084229, -0.669459879398346, 0.4765281081199646, 0.33721300959587097, 1.4522007703781128, -1.0643811225891113, -0.2971842885017395, -0.4561588168144226, -2.0956108570098877, 0.4509274661540985, 1.5720272064208984, -0.337566614151001, -0.04452093690633774, -0.8364463448524475, 0.30903515219688416, -1.0761581659317017, 0.09533566236495972, 0.9800471663475037, -1.762925386428833, -1.5303400754928589, -1.2667263746261597, 1.347975492477417, -0.7111992239952087, 0.2690070867538452, -0.23142148554325104, 0.8689818978309631, 1.1698522567749023, 0.5708653926849365, 0.40364184975624084, 2.5763890743255615, 1.0174667835235596, -0.535459578037262, 0.4928051233291626, -0.6629870533943176, -0.3860147297382355, 0.15248017013072968, 0.0387493371963501, 1.0349479913711548, -1.0196536779403687, -0.32516562938690186, -1.0876975059509277, 0.07250159233808517, -0.5860671401023865, -1.1025136709213257, 0.828947126865387, 0.6836267709732056, -1.092626929283142, -1.1683565378189087, -0.24534180760383606, 0.19522951543331146, -1.0233428478240967, -2.2571215629577637, 0.8497107028961182, 1.3458504676818848, -1.228447675704956, -0.9421247839927673, 0.3015548884868622, 0.39096251130104065, -1.7041141986846924, -0.03454393148422241, 1.812890648841858, 0.3710429072380066, 0.5822187662124634, -0.2794019877910614, 1.3197616338729858, 0.6471614241600037, 0.6009700298309326, -0.5946158170700073, 0.1185346320271492, 2.253103017807007, 0.6868705749511719, -1.6038206815719604, 0.9367493987083435, -0.39076530933380127, 1.0927069187164307, -0.8520700335502625, -0.3329525291919708, 0.618342399597168, -0.18565917015075684, 0.1712718904018402, 0.3468805253505707, -1.7642490863800049, 0.16312095522880554, -0.6333218812942505, 0.18653562664985657, 1.408974528312683, 0.04391731694340706, 0.25837939977645874, 0.8571838140487671, -1.4402588605880737, -1.7191096544265747, -1.1565873622894287, -1.0689184665679932, -1.3403316736221313, -0.3922428786754608, 1.0309346914291382, 0.0456974133849144, -0.5658119916915894, 0.6483533978462219, 0.6644386649131775, -1.8228639364242554, -0.16025924682617188, -0.7929885387420654, -0.14086829125881195, -0.19982849061489105, 0.8047910928726196, -0.20414189994335175, 2.1975882053375244, 0.8475709557533264, -1.6124871969223022, 0.5447397232055664, 0.6470291018486023, -0.29870256781578064, -0.033445123583078384, -0.23069562017917633, -1.1219055652618408, -1.4726788997650146, 0.2350541204214096, 0.6645607948303223, 0.47591352462768555, -1.150407075881958, -0.823046863079071, -0.40445706248283386, -1.935071587562561, -0.39495357871055603, -0.07576323300600052, -1.9395979642868042, 0.8098671436309814, 0.15565316379070282, 0.813930094242096, 0.703183114528656, -1.943740725517273, 0.21866674721240997, -0.33741262555122375, -1.3130333423614502, 0.7018538117408752, 1.1974992752075195, -0.35954999923706055, -0.8876788020133972, -0.16426941752433777, 0.22149622440338135, 0.6661801934242249, -0.46853893995285034, 1.3945494890213013, -0.5851592421531677, 0.376785010099411, -0.3306097686290741, -1.3799792528152466, 0.0682075172662735, 0.5828571915626526, -1.1438782215118408, 0.3670872747898102, 0.1266743540763855, -1.2669563293457031, -0.9352710247039795, 0.6041001081466675, -1.3582961559295654, -0.09343046694993973, -0.08226755261421204, 0.017014222219586372, 0.6752538681030273, 0.15634752810001373, 0.6377936601638794, 0.9671399593353271, -0.09094524383544922, 0.5973979830741882, -0.6576083302497864, 0.6141083836555481, 0.9588305354118347, 1.1867433786392212, -1.7083491086959839, -0.8798596858978271, 0.5767789483070374, 0.682478666305542, 0.015201176516711712, 0.5035887956619263, -0.14692862331867218, 0.3290698826313019, -0.08528836816549301, -0.861844003200531, -0.020891571417450905, -1.1028772592544556, -0.11031967401504517, -0.5770862102508545, 0.4372541308403015, -0.33212563395500183, -0.05587196350097656, -0.3446238338947296, -0.5049039125442505, 1.3044605255126953, 0.04360019415616989, -1.5612202882766724, 0.03127739578485489, -1.2262519598007202, 0.7267722487449646, -0.6710222363471985, -0.12481801211833954, 0.07319486141204834, 2.838426351547241, -0.6919057965278625, 0.7319849133491516, 0.26625388860702515, 1.2038613557815552, 0.34692132472991943, 0.04551840201020241, 0.6002640724182129, -1.3074488639831543, 0.5618636608123779, -1.6157673597335815, -0.7799882292747498, -0.2664337754249573, 0.313575804233551, 0.5624624490737915, -0.6107537150382996, -0.2920774519443512, -1.8395118713378906, -0.24962754547595978, 2.968337297439575, -0.3203512132167816, -0.8602179884910583, -0.10822207480669022, 1.0709718465805054, -1.0414263010025024, 0.02614840492606163, 0.06598047912120819, -1.2423509359359741, 1.1893761157989502, -1.2265253067016602, 0.1160818561911583, -0.15339748561382294, -0.331661581993103, -0.4591694474220276, -0.937029242515564, 2.377692699432373, -0.656152606010437, -0.1859835684299469, -0.18169452250003815, 0.2358100712299347, -1.335025668144226, -0.0657532662153244, -1.4494438171386719, -0.8229663372039795, -0.04432031139731407, 1.5173721313476562, 0.5549978017807007, -0.6056519150733948, -1.3903335332870483, -0.5493870377540588, -0.8615077137947083, -0.7306379675865173, -0.5590835809707642, 0.11811904609203339, 0.4396772086620331, 0.4856882691383362, 2.3686435222625732, -0.8250183463096619, 0.8691368103027344, -2.396474599838257, 0.2656247019767761, -1.574993371963501, -1.9067132472991943, 1.7934503555297852, -1.1949315071105957, -1.1111762523651123, -0.05746971815824509, 0.04780145362019539, 0.3314019441604614, -1.0918766260147095, -0.4437473714351654, -1.6166744232177734, -0.6289464235305786, 0.3042294383049011, 1.060916781425476, 0.21049091219902039, -2.240607976913452, -0.565009355545044], [-0.2952389419078827, -1.1675395965576172, -0.024384286254644394, 1.3864362239837646, 0.4398212730884552, 1.9352989196777344, -1.179572343826294, 0.7098644971847534, 0.633675754070282, 0.05754050612449646, -0.5062320828437805, -0.29406246542930603, -1.3644334077835083, 0.3962048590183258, 0.5263193249702454, -0.04640183225274086, 1.284441590309143, -0.48216643929481506, 0.26369747519493103, -1.7286969423294067, -1.3527809381484985, -1.7850229740142822, -0.1884842962026596, 0.4128812849521637, -0.5507967472076416, 0.5055369734764099, 1.047652006149292, -0.43775609135627747, -0.38667550683021545, -0.5441782474517822, 0.3703576624393463, 1.6058933734893799, -0.07093607634305954, -0.14469344913959503, -1.5469542741775513, -0.3776957094669342, -1.8942835330963135, -0.8712733387947083, -0.7792723774909973, -0.26407113671302795, 0.47191503643989563, -0.15772387385368347, 1.0492208003997803, -1.295764446258545, -1.8264787197113037, -0.11987467855215073, -0.6948696374893188, 1.778626561164856, 0.7899704575538635, -0.730040431022644, 1.0281883478164673, -0.4290156662464142, -1.1134861707687378, -0.34400251507759094, 0.9100832939147949, 0.9417874217033386, -0.06401394307613373, 0.11602221429347992, 0.1840813010931015, 1.2065093517303467, -0.726883053779602, 0.5234960913658142, -0.12225749343633652, 1.3336193561553955, -0.5692360401153564, -0.30693912506103516, -0.32196518778800964, -0.3173488974571228, -0.44720152020454407, -0.7265800833702087, 0.44082334637641907, -0.31146496534347534, 0.0002949422050733119, 0.08361748605966568, 0.670663058757782, 0.39830389618873596, -0.5358526706695557, 0.8804844617843628, 0.20263485610485077, -1.0293197631835938, 0.0639190748333931, 0.9192056655883789, 0.2677122950553894, 0.7113773822784424, 0.6949113011360168, 1.9942848682403564, 0.2878590226173401, -0.131766676902771, -0.4439127743244171, 0.0024316494818776846, 0.4619232714176178, 0.44109007716178894, 1.0212781429290771, 0.3638007640838623, 0.20338161289691925, -1.3859277963638306, -1.203514575958252, 0.12497792392969131, 0.3444952964782715, 0.1943139284849167, -0.8877245783805847, 2.3703370094299316, 0.958710789680481, 0.8265305757522583, -0.32996970415115356, -0.2547544836997986, 0.027998780831694603, -0.39231231808662415, 1.8297457695007324, 0.3727790117263794, 0.24361848831176758, 0.2887504994869232, 1.1326510906219482, 0.8934458494186401, 0.1474561244249344, 0.271405965089798, -0.2747519314289093, -0.3279670476913452, -0.7341636419296265, 0.20805096626281738, 0.3106003999710083, -0.3445811867713928, -0.3898656666278839, -0.6052905917167664, -2.016515016555786, -0.46692413091659546, -0.29024073481559753, 0.5667457580566406, 1.337231159210205, 2.156207323074341, -1.3472068309783936, -0.20006072521209717, 0.03929330036044121, -0.46670544147491455, 0.3852136433124542, -0.022215209901332855, -3.2458462715148926, -0.62552410364151, -0.10829804837703705, 0.5635337829589844, 0.8918659090995789, -0.3584675192832947, 0.2662736475467682, -0.6590137481689453, -0.2539841830730438, -0.9553769826889038, -1.3320690393447876, -0.05213655158877373, 0.5903481245040894, -0.9056415557861328, -0.14966167509555817, -0.01016557402908802, -0.030110910534858704, 1.7639840841293335, -0.3999074697494507, 0.7317187190055847, -0.2257198542356491, -1.5689467191696167, 0.594750702381134, -1.289472222328186, -0.3393406867980957, -0.600966215133667, -0.6414133906364441, 1.0600899457931519, -0.1844150722026825, -0.7135694026947021, 0.5721905827522278, -0.7235181927680969, 0.49622809886932373, 0.862348198890686, -0.5343523025512695, 0.804348886013031, 0.043780166655778885, -0.392760306596756, 2.0781960487365723, 0.3135203421115875, 0.20147384703159332, 0.734291672706604, 0.1457882523536682, -0.3953607678413391, 1.8632142543792725, 0.5474604964256287, -1.8467822074890137, -0.15518037974834442, 0.5296357274055481, -1.2749963998794556, -1.086549997329712, -0.33945462107658386, 0.3546448051929474, -0.4677705466747284, 1.034542441368103, -0.7097122073173523, -1.1840165853500366, -0.13660846650600433, 1.7173506021499634, 0.17022539675235748, 1.100289225578308, 1.4017572402954102, 0.42378875613212585, -1.863371729850769, -1.2656062841415405, 0.06419962644577026, 0.17226570844650269, -1.0447872877120972, -0.5590708255767822, -0.35544779896736145, 0.6228566765785217, 1.971545934677124, 0.5824028849601746, -1.075687050819397, 0.5449039936065674, 0.7542756199836731, 0.6387040615081787, -0.7637613415718079, 1.0617610216140747, -0.373735636472702, 1.4990806579589844, 0.6152500510215759, 0.1861322671175003, 1.0959657430648804, -1.1111778020858765, -0.5346260070800781, -0.2689240574836731, -0.26395583152770996, 1.7771782875061035, 0.49187034368515015, -1.9103081226348877, 1.795850157737732, -1.2890312671661377, -0.49445995688438416, -0.030837804079055786, 1.1665079593658447, 1.2858628034591675, 0.8142945170402527, 1.4706515073776245, -0.03139849379658699, -0.3329926133155823, 0.7748728394508362, 0.6801904439926147, -1.479643702507019, -0.26917675137519836, 0.6113129258155823, 0.13356198370456696, 1.026928424835205, -0.11339881271123886, -1.308968186378479, 0.4362860321998596, 0.8516363501548767, 1.6413028240203857, -0.2312639206647873, 0.5767316222190857, 0.15708203613758087, 0.5364106893539429, -1.3984465599060059, -0.33226248621940613, 0.8246386051177979, 1.7979021072387695, 0.24017196893692017, -1.3063430786132812, -1.909886360168457, -0.4311012029647827, 1.766302227973938, 0.8109511733055115, 0.37770307064056396, -0.5224393010139465, 0.779168426990509, -2.7925045490264893, 0.6804993748664856, -0.8975741267204285, 1.3672534227371216, -2.6903958320617676, -0.8414490222930908, 2.143690347671509, -0.5906082391738892, 0.3785395324230194, -0.10340467095375061, -0.5444321632385254, -0.49677684903144836, 2.5260446071624756, -0.5434838533401489, -0.42532482743263245, 1.7478060722351074, 2.301441192626953, -0.07613886892795563, 0.6213432550430298, -0.06354638934135437, 0.2352040559053421, -0.07356907427310944, 1.865845799446106, -1.7099034786224365, 0.017642322927713394, -1.4807687997817993, 0.3655867278575897, -1.895524501800537, 0.8104758858680725, -1.2841211557388306, 1.0371536016464233, 0.8083919882774353, -0.1352008432149887, -0.14374452829360962, 1.2447566986083984, 0.05413346365094185, 0.1620209813117981, -0.16274407505989075, 0.9070407748222351, -1.7592085599899292, 1.5053119659423828, -1.3829987049102783, -0.2523230314254761, -0.5567564368247986, -1.4526565074920654, -0.15789389610290527, 0.20851458609104156, 0.7703069448471069, 0.32445138692855835, 1.281721591949463, -0.5465186834335327, -0.413030207157135, 0.0969073623418808, -1.6616239547729492, 0.7301956415176392, -0.22035105526447296, 0.5688129663467407, -2.5276191234588623, 0.734106183052063, -0.03450712561607361, 2.2383873462677, -0.542337954044342, 0.07965222001075745, -0.432068407535553, -1.3050514459609985, 1.3381041288375854, -1.855501413345337, 0.11743732541799545, 1.2804601192474365, 2.428722620010376, -0.5350351333618164, 0.45419660210609436, 0.20800483226776123, 1.702656865119934, 2.4507999420166016, -1.4592736959457397, -0.831741988658905, 0.02934284880757332, 1.2573150396347046, 0.33863598108291626, 0.12070868909358978, 0.15488089621067047, 1.2213728427886963, -0.19805869460105896, -0.10297554731369019, -1.5090221166610718, -1.1548817157745361, -0.21628253161907196, -1.7684540748596191, -0.7702588438987732, 0.15666940808296204, 1.4562400579452515, -0.7586769461631775, -1.5672131776809692, -0.2891422510147095, -0.49640321731567383, -1.0683107376098633, 0.9469408392906189, 0.08946250379085541, -1.1706105470657349, 0.4771159589290619, -2.190803050994873, 0.8087931871414185, -0.07392514497041702, -0.06337635219097137, -0.5299047827720642, -0.049924954771995544, 1.1013386249542236, -0.1687798649072647, 0.9653221964836121, -0.24179692566394806, 0.5153593420982361, -0.1255846619606018, -0.14606456458568573, 1.5746819972991943, 0.8944576978683472, -1.5041836500167847, -0.19319520890712738, 0.3476508855819702, -0.9235142469406128, 0.7679236531257629, -0.6020908355712891, -0.5452089309692383, -0.8850116729736328, -1.4936243295669556, -0.15510594844818115, -0.23421958088874817, -0.4224378168582916, 0.9915804862976074, -0.20702651143074036, 2.6510207653045654, 0.6693434715270996, 0.07899383455514908, -2.3922622203826904, -1.167393445968628, 1.1467312574386597, -0.42076167464256287, -0.34439751505851746, -0.42281049489974976, -1.483601689338684, -1.291730284690857, -0.22048521041870117, 1.8486340045928955, -0.5965782403945923, -0.5551297068595886, 0.6763129830360413, 0.7640864253044128, -1.696658730506897, 0.843144953250885, 0.35535016655921936, -0.5426465272903442, -0.4793858826160431, 0.4054490327835083, 0.79627525806427, -0.05236329138278961, -0.7747474908828735, 0.09764453023672104, -0.37444329261779785, 0.4223647713661194, 1.825543999671936, -3.125814914703369, -0.16858814656734467, 0.8351709246635437, -0.6057199835777283, 0.04792403057217598, -0.9741170406341553, -0.33840179443359375, 0.2109357863664627, 1.271231770515442, -1.0278602838516235, 0.1352757215499878, 0.593172013759613, 0.9912510514259338, -1.5954328775405884, 0.2091762274503708, -0.018566234037280083, -1.1312769651412964, -0.5946012735366821, 0.6597323417663574, -0.194948211312294, 0.19132395088672638, -0.014855755493044853, 1.8110947608947754, -0.4443973898887634, -0.36553695797920227, -0.38129130005836487, -1.162212610244751, -1.2605022192001343, 0.7236043810844421, 0.6975181102752686, 0.7837963700294495, 0.1989947408437729, -0.4545668661594391, -0.9829196929931641, -0.8446274399757385, 0.48351094126701355, 1.9768996238708496, -1.80520761013031, 0.7939468026161194, -0.27508243918418884, 0.7406480312347412, -0.8901795148849487, -0.31799858808517456, -0.6701242923736572, 0.7772974967956543, -0.5786535143852234, 0.12987078726291656, 0.8431118726730347, 1.6704787015914917, 0.1272047907114029, 0.22767361998558044, 2.0738611221313477, -0.42828598618507385, -0.7136228084564209, -1.2760388851165771, 0.7398661375045776, 0.5960876941680908, 0.24865300953388214, 1.950115442276001, 0.07890158891677856, -1.6795148849487305, -1.167772889137268, 0.2508171498775482, -1.3787267208099365, 0.5830420255661011, -0.5614066123962402, 0.853321373462677, 0.13381442427635193, 0.43114542961120605, -0.03150225058197975, -0.3527097702026367, -0.44232282042503357, -0.5856176614761353, -1.3365050554275513, -0.2516986131668091, 0.7445068359375, -1.4012629985809326, 0.1663079410791397, -0.9176745414733887, 1.1341716051101685, 0.4376000165939331, 0.15050481259822845, -0.3980378210544586, -0.19608092308044434, -0.19427448511123657, 0.49830326437950134, 1.7393215894699097, -0.6152676343917847, -1.4358476400375366, 0.6252260208129883, 0.5946284532546997, 1.0285193920135498, -0.1739170402288437, -0.4631395637989044, 1.0002487897872925, -0.6063568592071533, -1.6643506288528442, 0.32043227553367615, 0.4443035423755646, 0.6518068909645081, 0.06697922945022583, -0.6795084476470947, 0.9167395830154419, 0.24162079393863678, -1.1431632041931152, 0.8139456510543823, -0.19132199883460999, 1.1921064853668213, 0.16861167550086975, -0.6753422021865845, -0.3048079013824463, -0.7684566974639893, -2.606279134750366, -0.195164293050766, -0.4498008191585541, 1.9206851720809937, 1.3502490520477295, 1.0144509077072144, -0.6205800175666809, 0.775881826877594, -0.15391720831394196, -2.0572595596313477, -3.998326063156128, 0.00890953466296196, 0.996573805809021, -0.17213118076324463, 1.962493658065796, -1.0939124822616577, 0.23903755843639374, -1.7981715202331543, 0.532906711101532, 2.129686117172241, 1.388706922531128, -1.241302490234375, 2.1833512783050537, 0.1136622205376625, -0.0808098167181015, 0.278939425945282, 0.9517855048179626, -1.1215943098068237, -0.2650756239891052, 1.5481009483337402, -1.0491721630096436, -1.2643766403198242, -1.304874062538147, 1.6912798881530762, 0.20997770130634308, 0.433522492647171, 1.2695508003234863, -1.714380145072937, 1.5277830362319946, -0.361783891916275, 0.3152300715446472, 0.9037317037582397, -0.0003517769800964743, -0.08606059849262238, -0.3940669596195221, 0.7577124238014221, -0.27380669116973877, -0.2765772342681885, 0.7340851426124573, -0.540911853313446, -1.345572829246521, -1.3168435096740723, -0.7239114046096802, -1.1617597341537476, -0.5297681093215942, -1.259506344795227, -0.7841609716415405, 0.3861616253852844, 0.8301972150802612, 0.07396060973405838, 0.8697689771652222, 0.01586482673883438, -0.721061646938324, 0.015529481694102287, -3.498969554901123, -0.7915435433387756, -0.9173334240913391, -0.8595436215400696, 0.1042315885424614, -1.515814185142517, 2.057898759841919, -1.1504716873168945, 0.8975452780723572, -0.9428547620773315, 0.20660924911499023, 0.033921509981155396, -0.20238259434700012, 0.2787837088108063, 1.0321518182754517, 0.1388109028339386, 0.3957892656326294, -1.1255172491073608, 1.4784302038606256e-05, -0.5016466379165649, -0.4636802077293396, 0.03809326887130737, 0.9427369832992554, 0.8164942860603333, 1.302615761756897, -0.2324357032775879, -0.5217283368110657, 0.45147091150283813, 1.3979178667068481, -1.0715020895004272, -0.4425561726093292, 0.2281423956155777, 0.30536746978759766, 0.12223204225301743, -0.16499033570289612, -0.8909991979598999, 2.2154037952423096, 1.095579981803894, -0.040137600153684616, -0.2192753106355667, -0.9367809295654297, 1.0825929641723633, 1.0698561668395996, -0.46803420782089233, -0.45673561096191406, -1.1121727228164673, -1.194204568862915, -0.2486475110054016, -0.08584191650152206, 0.7348554134368896, -1.4607093334197998, 0.7688835859298706, 0.491936594247818, 0.6854569911956787, 0.3510589599609375, 0.8016964197158813, -0.5310707092285156, 2.032365322113037, -0.3035944402217865, -2.02294921875, 0.9964368939399719, 0.883388876914978, -0.2666735351085663, 0.7011411190032959, -0.5894794464111328, 0.43733417987823486, 0.307536244392395, -1.345560908317566, 0.42073380947113037, -0.4273488521575928, -0.8763535618782043, 0.8956657648086548, -1.107146978378296, 1.7959864139556885, -1.5338410139083862, 0.06671290099620819, 0.155399352312088, -0.4121270775794983, -1.784924030303955, -0.6925828456878662, -1.4168682098388672, -1.0888164043426514, -0.06398436427116394, 0.16947366297245026, 0.686255156993866, -0.8326672315597534, -1.4469143152236938, 0.05788857489824295, 0.22800371050834656, -0.24048088490962982, 0.5983655452728271, 0.7872226238250732, -0.2868686616420746, -0.1258237063884735, -1.1924750804901123, 0.8970062136650085, -0.3279898762702942, -0.8187268376350403, 0.32720470428466797, -0.2488366961479187, 1.3659518957138062, 1.606878399848938, 1.1426054239273071, -1.7552176713943481, -1.4292739629745483, 0.033759016543626785, 0.4905334711074829, -1.3055768013000488, 0.8313118815422058, -0.2729337811470032, 1.2837116718292236, 0.720184862613678, 0.11403963714838028, 0.08649366348981857, 1.3784644603729248, 0.47379040718078613, 0.7148167490959167, 0.23159672319889069, 0.22334745526313782, -0.6969436407089233, -1.16252863407135, -0.18440519273281097, -1.1198011636734009, 0.3704065978527069, -0.3115760087966919, 0.9758034348487854, -0.4604388475418091, 2.824256420135498, -0.32995009422302246, 1.7417031526565552, -0.9159800410270691, 0.39557084441185, 0.5552830696105957, 1.0786484479904175, -2.063164472579956, 0.2906620502471924, -0.2962592840194702, 0.4320243000984192, -1.2875003814697266, 1.5290402173995972, -0.5846643447875977, -0.027582615613937378, 1.0670596361160278, -0.023519931361079216, 0.8106743693351746, -0.18453210592269897, 0.9645137786865234, -0.7577204704284668, 0.5308066010475159, 0.6896421313285828, -0.19699250161647797, 0.6544890403747559, -1.3704818487167358, -0.5747838020324707, 1.4455111026763916, -0.17370733618736267, 1.030448079109192, 0.9072957038879395, 1.4815257787704468, 0.07829102128744125, -2.5558111667633057, 1.7471286058425903, 0.5034316182136536, -0.03315902501344681, 0.7409965991973877, 0.2121839076280594, -0.7399507761001587, -0.4952152967453003, 2.3312036991119385, -0.4131115972995758, -0.42522895336151123, -0.23657049238681793, 0.35538309812545776, 0.05198025703430176, -0.450944721698761, -0.868786096572876, 0.5302430987358093, 0.39389121532440186, 0.222405806183815, 0.9214521646499634, -1.1762166023254395, -1.8607912063598633, -0.7693760991096497, -1.7354925870895386, -0.12783795595169067, 1.08036208152771, -1.1579781770706177, -0.41000184416770935, 1.2808862924575806, 1.0006650686264038, 0.9478719830513, -1.9789743423461914, 1.6918113231658936, -0.4473184049129486, -0.8564493656158447, 0.5552893280982971, 1.0202847719192505, 0.712748646736145, 0.37759894132614136, 0.2977369427680969, 0.012176151387393475, 0.46802476048469543, -1.1241601705551147, -1.9491840600967407, 1.3727924823760986, -0.16673734784126282, -0.4441204071044922, -0.03997040167450905, -1.000215768814087, 0.1818298101425171, 0.8748031854629517, 0.02716917358338833, 0.423944890499115, 1.8568341732025146, -0.32234644889831543, -0.21031810343265533, 0.09878178685903549, 0.557593047618866, 0.6811894178390503, -0.1016850546002388, -0.5445007681846619, -0.7604330778121948, 0.8645170331001282, -0.30993521213531494, -0.31462690234184265, 1.4062964916229248, -0.8044263124465942, 0.13317252695560455, 0.3377220928668976, 0.7188383936882019, -0.5172414183616638, 1.217715859413147, 0.42977914214134216, 0.42943108081817627, 0.5258576273918152, -2.0473318099975586, 1.011566400527954, 0.9248698353767395, 0.28949135541915894, -0.9871122241020203, -0.11404134333133698, -0.11360104382038116, 0.31265681982040405, -0.07329061627388, 1.0066746473312378, -0.48925381898880005, 0.15498170256614685, -0.06131916493177414, 1.9720995426177979, -1.8501331806182861, 0.8541821241378784, -0.36364349722862244, 0.14157523214817047, -0.4834427237510681, 0.5771875381469727, 1.4291671514511108, 0.9136791229248047, -1.9791518449783325, -0.8062795996665955, -0.3850161135196686, 1.4300379753112793, 0.9338241219520569, -1.3052325248718262, 1.5160744190216064, 1.9679898023605347, -0.955191969871521, 1.3692766427993774, -0.7169256806373596, 1.9328341484069824, 0.09829078614711761, -0.9354285597801208, -0.08057013154029846, -1.07478666305542, 0.5269978046417236, 0.11216644942760468, 0.26506370306015015, -1.1340652704238892, -0.40990546345710754, 1.3655539751052856, -0.0404069647192955, 0.6504368185997009, -1.0277905464172363, -0.2500634789466858, 0.5260891914367676, -0.46633216738700867, -0.5225694179534912, 1.708842396736145, -1.112300157546997, 1.758589267730713, -0.10554095357656479, 0.021953873336315155, 0.3979845643043518, 1.4959055185317993, -1.7889219522476196, -0.5192241668701172, 1.0617902278900146, 0.6551768183708191, 1.0366086959838867, -0.14207862317562103, -0.5574719309806824, 1.0127302408218384, -0.39578545093536377, -2.091723918914795, -0.1698039323091507, -1.3335001468658447, 0.36297133564949036, -0.8641138672828674, 1.385116457939148, -0.967391312122345, -1.5839297771453857, 0.3063845634460449, 0.703552782535553, -2.4481074810028076, 0.4378105103969574, -0.28819844126701355, -0.3324180543422699, -1.7762460708618164, -0.6749083399772644, 0.443594753742218, 0.5246970653533936, -0.3675222098827362, -1.1893888711929321, 1.0551904439926147, -0.5286359190940857, 1.6185932159423828, 0.6769137978553772, 0.5944100022315979, -0.3890218138694763, -1.9099698066711426, -0.9303662180900574, -0.5990710854530334, -0.4843825101852417, 0.473392516374588, -0.4647250175476074, -0.6211658716201782, 0.19906631112098694, -2.571646213531494, 1.26766836643219, -0.13119395077228546, -0.05265989527106285, 0.3765178918838501, -0.3343915343284607, -0.8598840236663818, -1.0813148021697998, 1.0402644872665405, -1.0758622884750366, 0.36299991607666016, 0.8854528665542603, -0.5125612616539001, -0.6287717223167419, 0.10737181454896927, 1.2755991220474243, 1.7395832538604736, -0.8555645942687988, 0.5203378200531006, -1.5056042671203613, -1.1589971780776978, -0.4426797926425934, 1.4424567222595215, -0.491750568151474, -1.7328381538391113, 1.0420904159545898, -1.1265335083007812, 0.6955368518829346, -0.19979512691497803, -0.4272369146347046, -0.8880299925804138, 0.9762821793556213, -0.6818603873252869, 0.504208505153656, 0.5235404968261719, -0.08432606607675552, -1.0637376308441162, 1.2796059846878052, -0.32613685727119446, 0.7812803387641907, -0.10537361353635788, -0.025996757671236992, 0.10975619405508041, -1.0879994630813599, 1.194735050201416, -0.362855464220047, 1.2052772045135498, 0.11592603474855423, -0.18253354728221893, 0.3873964846134186, 0.24146020412445068, 0.3209185302257538, 0.9297512173652649, -1.0249271392822266, -0.45343345403671265, 1.0909411907196045, 1.0442556142807007, -1.8855187892913818, 0.7444503903388977, -0.6387923359870911, 0.22878755629062653, -0.6051029562950134, 0.7380343675613403, -0.4949733018875122, -0.025379784405231476, 0.005390147212892771, -0.038976918905973434, 0.08508535474538803, 0.7417004704475403, 1.3964364528656006, -0.3129923343658447, 0.9191284775733948, -1.4798988103866577, -0.2155725359916687, -0.12559078633785248, 0.5042871236801147, 0.9054126143455505, 0.08811821788549423, 0.5177220106124878, 0.972586452960968, 0.5257636308670044, 1.0630481243133545, 0.6969266533851624, -0.20790113508701324, -0.395092248916626, 0.674656867980957], [0.48233747482299805, 0.8133755922317505, -0.7121779322624207, -2.0025980472564697, -0.05194741114974022, -0.24952515959739685, -0.5385351777076721, 0.250904381275177, 0.16028133034706116, -0.7711330056190491, 0.29807206988334656, -0.020532507449388504, 0.26921042799949646, -0.03247413784265518, 0.6042537689208984, -1.0506006479263306, -0.9250423908233643, -0.8852167129516602, -1.2645025253295898, 1.6186597347259521, -0.709688127040863, -1.9385347366333008, -0.8858346939086914, -0.3379858136177063, 0.9412679076194763, 0.6714202761650085, -1.895943284034729, 1.0273091793060303, 1.5315173864364624, -0.37674480676651, 0.5100551247596741, 0.4510616958141327, -2.1665279865264893, -0.4869575798511505, 0.564976692199707, 0.16852188110351562, 0.730746328830719, 0.5077232718467712, 1.4506901502609253, 1.7446178197860718, 2.172236442565918, 0.44147229194641113, 1.4417140483856201, 0.8648118376731873, 0.5014368295669556, 0.8618426322937012, 1.0652062892913818, -1.0512137413024902, 0.22679503262043, 0.04698754847049713, 1.330199122428894, 0.08367053419351578, -0.6277584433555603, -1.5342352390289307, 1.7180290222167969, -1.3643333911895752, 0.1693136990070343, 1.6471065282821655, -0.7044826745986938, -0.31313639879226685, -0.9818658828735352, -0.42658501863479614, -0.4464751183986664, -0.4564834237098694, 1.4568567276000977, 0.8727798461914062, -0.5263643860816956, -1.6715290546417236, -1.970679759979248, -1.2910305261611938, -0.24678383767604828, -1.397519588470459, 0.6321918964385986, 0.6764772534370422, -0.48760148882865906, 2.230487585067749, 0.21833321452140808, -1.4184041023254395, -0.31930917501449585, -1.1704257726669312, -0.9174058437347412, -0.12513548135757446, -1.730353593826294, 0.2679443359375, -0.37306368350982666, -0.9325684309005737, 0.5587003231048584, -0.18053068220615387, 2.2459921836853027, 0.3394522964954376, 0.23166553676128387, 0.8369094729423523, 0.6446097493171692, 0.09625744819641113, -0.7414355874061584, 0.2920604944229126, -0.6900292634963989, -0.02149185910820961, -2.7763848304748535, 0.9203746914863586, 0.12029246985912323, -0.3050312101840973, 1.2881122827529907, 1.4051929712295532, -0.3900385797023773, -1.0321125984191895, -1.8030232191085815, 0.398340106010437, 0.22081556916236877, 0.9259551763534546, -0.7044503092765808, 0.3367744982242584, -1.2888141870498657, 0.3435956835746765, -0.8595357537269592, 0.03095570206642151, 0.5814668536186218, 0.3755141496658325, 0.9614483714103699, 0.5519969463348389, 0.8555274605751038, -0.04674074426293373, 0.3925648331642151, 1.0269062519073486, 1.2781587839126587, 0.9545813202857971, 1.169100284576416, 0.15250642597675323, -1.4431074857711792, -0.015869982540607452, 1.2040200233459473, 0.7886814475059509, 0.32807812094688416, -0.8730322122573853, 1.1083513498306274, -0.9143787622451782, -0.17761516571044922, 0.8594875931739807, 0.1871122419834137, -0.3296257555484772, 0.4173220992088318, -0.7296718955039978, -0.34276407957077026, 0.8381356596946716, -2.1319739818573, 0.709577739238739, -0.5763853192329407, -0.3213620185852051, 0.652779221534729, -1.0865037441253662, -1.2908520698547363, -1.925432562828064, 1.9424422979354858, 0.10863007605075836, -0.024586664512753487, -0.2982058823108673, 1.4247276782989502, 0.6782911419868469, 0.4907239079475403, -0.6655354499816895, 0.7240970134735107, 1.2453298568725586, 0.1643720418214798, -1.0621007680892944, -1.7273815870285034, -0.42803889513015747, 0.22715707123279572, 0.03786088153719902, -0.8355507850646973, 0.7790942192077637, 1.6669005155563354, -1.3560891151428223, 0.9621949195861816, -1.0767167806625366, 0.5709361433982849, -0.3580499291419983, 0.20693203806877136, -0.4517512321472168, -0.7711228132247925, 0.07457154989242554, -0.24740906059741974, 0.6290259957313538, 0.07357300072908401, 1.7162730693817139, 0.06729261577129364, -0.5336878895759583, 0.4293750524520874, -0.12479633837938309, -0.2725728154182434, 0.7109901309013367, -0.9432687759399414, 1.8565208911895752, -0.9543878436088562, 1.5650407075881958, -0.5132579803466797, 0.00691003305837512, 1.415859341621399, -0.7375196218490601, -2.543898105621338, -0.13577812910079956, 0.6403853297233582, -0.7825995683670044, -0.5745893120765686, 0.4074367880821228, 2.3186256885528564, 1.6936230659484863, 0.49654534459114075, -0.6221702694892883, 0.9439960718154907, 1.9042446613311768, -0.6905107498168945, 0.7261306047439575, 0.8348608613014221, -1.1813138723373413, 1.2540531158447266, 0.6242005228996277, 1.998024344444275, 0.8399466276168823, 2.0942273139953613, -0.39045748114585876, 0.19096249341964722, -0.016567017883062363, -0.6619824767112732, -1.42387056350708, -0.11369648575782776, 1.2311137914657593, 1.0112756490707397, -0.9488800168037415, 0.5666617751121521, 1.1217381954193115, 0.6131709218025208, 0.2127583771944046, 0.6699718236923218, 0.2722335755825043, 1.2151449918746948, 1.160449504852295, 1.4113922119140625, -0.061268407851457596, -1.188421607017517, -0.17582321166992188, 2.657870054244995, -1.2031372785568237, 1.5302464962005615, -1.3952471017837524, -0.06040381267666817, -2.7815513610839844, -0.7774035930633545, 2.4920592308044434, 0.6823442578315735, -0.390377402305603, -1.6620615720748901, 0.6041358709335327, -1.180988073348999, 2.4671175479888916, -1.4384416341781616, 1.0435534715652466, 0.6653338670730591, 0.2931863069534302, -0.17369477450847626, 1.1731699705123901, 0.34504935145378113, -0.7188205718994141, 0.1925571858882904, -0.9561692476272583, 0.35387036204338074, -1.0974162817001343, 0.07768522948026657, 0.28526589274406433, 1.7433522939682007, -0.1697475165128708, -1.3366217613220215, 2.448312759399414, -0.021588653326034546, -0.8232431411743164, -0.7869923710823059, -0.07602481544017792, -0.5184768438339233, 0.2692028880119324, 0.3397067189216614, -1.032135009765625, 0.02245546318590641, -0.39872997999191284, -0.9914295673370361, 0.3492722511291504, 1.7073657512664795, -0.9761250615119934, -0.4537684917449951, 0.08519323170185089, -2.6225664615631104, -0.639428436756134, -0.7807790040969849, -0.6092730164527893, 0.504097580909729, 1.1337623596191406, -0.4067937135696411, 0.4132640063762665, 1.8071569204330444, 1.0703282356262207, -0.26227501034736633, 0.40603598952293396, 0.8648239374160767, 1.296844482421875, -0.988300621509552, -0.4199592173099518, -1.9929134845733643, -0.7766479849815369, 1.4807511568069458, 0.3215314447879791, 0.10094719380140305, 0.9347620010375977, -0.3232697546482086, 0.4909422695636749, 1.1679847240447998, -0.5885649919509888, 0.011488375253975391, 0.41404256224632263, 1.5848959684371948, 2.694300889968872, -0.09878750890493393, -0.2888382077217102, 0.6884829998016357, 0.6804280281066895, 2.034156322479248, -0.21469324827194214, 1.8305518627166748, 0.8754808902740479, 0.8689885139465332, -0.25595152378082275, 0.79084312915802, 1.1772733926773071, -0.5024945139884949, -0.9851565361022949, 0.5176739692687988, -0.8318552374839783, -0.6030582785606384, 1.2935495376586914, 1.3643898963928223, 0.5651749968528748, -0.8582251071929932, -0.9643141031265259, 1.2145085334777832, -0.8871588706970215, 0.15389905869960785, 0.3404678702354431, -1.7142918109893799, 0.8548451662063599, -0.3441215753555298, 0.31802359223365784, 0.10865004360675812, 1.0718289613723755, -0.2288791984319687, 0.36898139119148254, -0.6608755588531494, 2.2839457988739014, -1.1186614036560059, 0.2699442505836487, 0.5041171908378601, -0.4826063811779022, 0.6796575784683228, 1.8294379711151123, 2.510840892791748, 2.8431460857391357, 0.12289145588874817, -0.06620296835899353, 0.2117253839969635, -0.5622620582580566, 0.8119372725486755, 0.5009676814079285, -0.5246907472610474, 0.5857385993003845, 0.8977612853050232, -0.7679598331451416, 0.10574662685394287, -1.0692251920700073, 0.6686322689056396, -0.811003565788269, -0.4929085671901703, 0.4393971264362335, 0.2654722034931183, 0.42342936992645264, 1.8339051008224487, 0.946518063545227, -0.278087317943573, -0.5977296233177185, -0.9109610915184021, 0.08146025985479355, -2.654595136642456, 1.5502885580062866, -0.16372248530387878, -0.5351366400718689, -0.3847467005252838, -1.9293663501739502, -0.5188467502593994, 0.3097028434276581, 0.24965324997901917, -1.373605728149414, 0.03803170472383499, 0.11878816783428192, -0.481840044260025, 1.2637221813201904, 2.2728829383850098, -0.9978629350662231, 0.866402804851532, -0.050351861864328384, -0.0712224692106247, -0.3317280113697052, -0.9002079963684082, -0.013562030158936977, 0.07579291611909866, 0.972101092338562, -1.1236963272094727, -0.46195173263549805, 1.440354347229004, 0.35720300674438477, -0.016090888530015945, -2.1587753295898438, 1.125787377357483, 0.7164193987846375, 0.8637775182723999, -1.0598182678222656, 0.7584633827209473, -0.4449293911457062, 0.5861915349960327, 0.43561115860939026, -0.36329734325408936, 0.48952239751815796, -0.2438174933195114, 1.579949975013733, 1.005507230758667, -0.35219696164131165, 0.40867552161216736, -0.4752677083015442, -0.6084333658218384, 1.2647581100463867, -1.0037249326705933, -0.4698467254638672, -0.5208061337471008, -0.3418301045894623, -1.9031901359558105, -1.9808118343353271, 0.30110716819763184, 0.24242080748081207, -0.009679784998297691, 0.36763909459114075, -0.32412347197532654, -0.5915228724479675, 0.5224621295928955, -0.47544723749160767, 1.444548487663269, -1.0398494005203247, -1.0182582139968872, -0.2053050398826599, 0.2415984869003296, -0.5063109993934631, -0.06592562049627304, -1.2979600429534912, -1.1276957988739014, -0.08978945761919022, 1.5411527156829834, -1.312963843345642, -0.5558350682258606, 1.2369844913482666, 0.9727191925048828, 1.5812931060791016, 0.21196019649505615, -1.1577000617980957, -0.16113947331905365, 1.7976535558700562, -2.026916980743408, -0.3185715973377228, -0.3623783588409424, -1.3082441091537476, 0.7861312031745911, -0.9342749714851379, -1.3263254165649414, -0.39178574085235596, -0.16859887540340424, -1.0462485551834106, 1.4598243236541748, -0.6282083988189697, 0.6051459908485413, 0.9008885622024536, 2.2144970893859863, -0.6834354996681213, 0.8398826718330383, 0.04299534484744072, -0.4979051649570465, 2.813035249710083, 0.43595317006111145, -0.8019150495529175, -1.2587416172027588, 0.45262572169303894, -0.6489784717559814, -0.40362074971199036, 0.5509397983551025, 0.9020019769668579, 0.9339649081230164, 1.1835302114486694, 1.9718695878982544, 1.659995675086975, -0.3791837990283966, -1.7220947742462158, 0.4448557198047638, 1.2767927646636963, 2.1393074989318848, 1.0865336656570435, 0.5604345202445984, -0.38415563106536865, -1.3242937326431274, -0.9159116148948669, 1.2948558330535889, 1.8350396156311035, -0.8897200226783752, 0.2626626491546631, 1.3607051372528076, -1.4919813871383667, 1.7757755517959595, 0.026147592812776566, 0.825282096862793, 0.36151883006095886, 0.8414788246154785, 1.2218166589736938, 1.6094945669174194, 0.4803432822227478, -1.3562731742858887, -0.9039755463600159, -0.9195907711982727, 1.5739853382110596, -1.0887283086776733, -1.308316946029663, -0.7617653608322144, -0.2867647409439087, 0.09202051907777786, -1.1882432699203491, -0.056010399013757706, 1.1448869705200195, -1.6030144691467285, 0.2681364119052887, 0.027603501453995705, 0.8413178324699402, 1.408133864402771, -1.126306414604187, -0.32899925112724304, 0.3147251307964325, 1.3408279418945312, -1.1482843160629272, -0.07685084640979767, -0.7878837585449219, 0.795324981212616, -0.47451916337013245, -0.9495534300804138, -1.3875858783721924, -0.13440074026584625, 0.5883391499519348, -0.5200889110565186, 2.3836467266082764, -1.552303433418274, -0.9679589867591858, 0.22757123410701752, 0.21815212070941925, 1.1239898204803467, -0.5390869379043579, -0.449077308177948, 1.207392692565918, 2.082388401031494, -0.03373169153928757, -0.3286100924015045, 0.6607804894447327, 2.178917646408081, 1.297370195388794, -1.1601612567901611, 1.2962357997894287, 0.14447622001171112, -0.9072030186653137, 1.747857928276062, -1.4344974756240845, 1.0047193765640259, -0.9650505185127258, 1.3188414573669434, 0.1218394860625267, -0.2972491681575775, -0.33881956338882446, -0.49648934602737427, 0.47114524245262146, 1.3609009981155396, -1.3463212251663208, 0.9363070726394653, 2.346602201461792, -0.7970331907272339, 1.1269197463989258, -2.0750763416290283, 1.0520429611206055, -0.3681091070175171, -0.5599538683891296, 1.220112681388855, 0.8377277851104736, -0.7397326231002808, 1.1732473373413086, 0.18387091159820557, 1.3079205751419067, -0.47519344091415405, -0.6632282137870789, 0.1473347246646881, 0.12520550191402435, -1.1716628074645996, 1.026025414466858, -0.1347762495279312, 0.8017688393592834, -0.33718350529670715, 0.46633636951446533, -1.422974944114685, -0.30330896377563477, 0.18081337213516235, 0.659748911857605, 1.116662621498108, -0.048366934061050415, -0.21449598670005798, -0.31961503624916077, 0.8405450582504272, 0.8816359043121338, -1.803924322128296, -1.9937206506729126, 1.8746685981750488, 0.748657763004303, 0.25468143820762634, 0.6111947298049927, 0.8389825224876404, 0.7947335243225098, -0.036119244992733, -1.4905754327774048, -0.21552355587482452, 1.21761155128479, 1.1611710786819458, 0.2100546509027481, -1.0472383499145508, 0.21679085493087769, 0.06914596259593964, -2.8414177894592285, 0.48501211404800415, -1.3456284999847412, -1.5525418519973755, -0.3964204490184784, -0.1887815147638321, 0.7969483733177185, 1.301496148109436, -1.7447798252105713, 1.3045964241027832, 0.3465076684951782, 0.8700181245803833, 1.7315614223480225, -0.5119599103927612, 0.40908658504486084, 0.8426986932754517, 0.8850144147872925, -0.14748340845108032, 0.09927340596914291, 0.3615119457244873, 1.7513917684555054, -0.651289165019989, -1.160944938659668, -2.159043788909912, 1.2960805892944336, 0.03775400295853615, -0.20100991427898407, -0.37278732657432556, 1.0672614574432373, 0.29544758796691895, -1.3226304054260254, -0.4132300913333893, 0.010169838555157185, -1.325443148612976, -0.061502814292907715, 0.3391110599040985, 0.029309691861271858, -0.39971598982810974, -0.0740659087896347, -1.945352554321289, 1.4156574010849, 0.7233713865280151, 0.1118372231721878, -0.5271064043045044, -0.22727498412132263, -0.5236621499061584, 0.23735931515693665, -0.009769836440682411, -0.6876484751701355, 0.5345455408096313, 1.3439205884933472, -0.02274726703763008, 0.5261127352714539, 0.38485729694366455, -0.6854164004325867, -0.8630734086036682, -1.0853554010391235, -0.2624913156032562, 0.8435347676277161, -1.560367465019226, -1.6826270818710327, -1.3636412620544434, 0.28372687101364136, -0.8660393357276917, 0.8395591378211975, -0.009935955516994, 0.1523808091878891, 2.5208330154418945, 0.09502692520618439, -0.44106292724609375, -0.4179873764514923, -0.3720449209213257, -0.9418050646781921, 0.6225936412811279, 0.7121713161468506, -0.5402641296386719, 1.525041937828064, 1.075847864151001, -0.06744446605443954, -0.39518919587135315, -0.7580419778823853, 0.6448326706886292, 1.1236764192581177, 0.5269071459770203, 2.467759609222412, -0.017621219158172607, -0.007412932347506285, -0.8005523085594177, -0.19937776029109955, -0.2074051946401596, -2.144761800765991, -0.3264904320240021, -0.2318921834230423, 0.2674838602542877, -0.19795309007167816, 0.04871327057480812, 0.33883753418922424, 1.1396045684814453, -0.4329792857170105, -1.0712848901748657, 0.22570106387138367, 0.7065901160240173, -0.6292880177497864, -2.778125762939453, -0.3095241189002991, 0.4754446744918823, 0.317200243473053, -0.06907972693443298, 0.037836704403162, 0.41131046414375305, -0.3885571360588074, 1.7291098833084106, -0.48807433247566223, -0.13212338089942932, 0.09282242506742477, 0.16857966780662537, -1.899471402168274, 1.3255678415298462, -0.592219352722168, -0.3730787932872772, 1.7790204286575317, -0.19521063566207886, -1.2592614889144897, -0.10796088725328445, -1.2535598278045654, 0.8282836079597473, -1.4462794065475464, 3.4235801696777344, 0.31664615869522095, -0.8182198405265808, 1.0457305908203125, -0.023740995675325394, 1.3028143644332886, 0.4994240999221802, -1.2520841360092163, -0.15125590562820435, -0.18798889219760895, -0.785706639289856, -0.6078734993934631, 1.4017964601516724, -0.4471434950828552, -0.8681778311729431, -0.2462945282459259, 0.29899078607559204, 0.6218633651733398, 0.627299964427948, -1.3724478483200073, 0.9790134429931641, 0.062366969883441925, 0.32956036925315857, 0.3847351670265198, 0.6261383891105652, -0.6526846885681152, -1.9574776887893677, 0.9809660315513611, 0.9462099671363831, -0.6201691031455994, -2.0900790691375732, -2.0196495056152344, 1.7282741069793701, 0.12122344970703125, -0.6078837513923645, 0.6221928596496582, 1.1507869958877563, 0.03400256112217903, 0.029858628287911415, -0.029246799647808075, 1.1458017826080322, 0.926559567451477, 1.225278615951538, -2.083958864212036, -1.529401183128357, 0.09834989160299301, -0.6567495465278625, -0.6435990333557129, -0.545690655708313, 0.5105118155479431, 0.054313190281391144, 0.38056549429893494, -0.2809945344924927, 1.087383508682251, 1.429579734802246, -1.2117716073989868, -0.16981928050518036, -0.9898518919944763, -0.3055272400379181, -0.6047769784927368, 0.7310009598731995, -0.7050008773803711, 0.02089054509997368, -0.5118822455406189, 0.05772127956151962, -0.5488265156745911, -0.3470483124256134, -0.18639427423477173, 0.3468085527420044, 0.42971280217170715, -1.6584300994873047, -0.039971329271793365, 0.6350487470626831, 0.12967027723789215, -1.0330250263214111, 1.015241026878357, -1.6134684085845947, -0.4522024989128113, -1.0408203601837158, -2.1344828605651855, -1.0847563743591309, -0.11720945686101913, 0.17614978551864624, 0.9380011558532715, -0.19257840514183044, -0.6001753211021423, 0.07962445169687271, -0.02556891366839409, -0.8866782784461975, -0.3006056249141693, 1.9905080795288086, 1.1455650329589844, -0.7202606797218323, 0.2093379944562912, -2.058997869491577, -0.02734626829624176, -0.3775493800640106, 0.012450399808585644, 2.206963300704956, -1.188097596168518, -0.35097816586494446, -1.1976310014724731, 0.024540409445762634, 0.36515921354293823, -1.2298645973205566, -0.6374309659004211, 1.252791404724121, 0.9032294154167175, 1.8276649713516235, 0.08794235438108444, 0.7478795051574707, 0.2203376591205597, -0.6342808604240417, 1.2982407808303833, -0.14524878561496735, 0.8308207988739014, -0.9421022534370422, -0.75129234790802, 0.9601376056671143, -1.1780258417129517, 0.8815807700157166, 0.061122387647628784, 1.1424548625946045, -1.027540922164917, 1.1281960010528564, -1.9080995321273804, -0.8132146596908569, 0.7429324388504028, 0.7321265935897827, -0.700717568397522, 1.141959309577942, 0.46086713671684265, 0.5213747620582581, 1.40288245677948, -0.7857364416122437, -1.2802711725234985, 1.6243187189102173, -0.8801503777503967, 0.5738584399223328, -0.8897186517715454, -0.354823499917984, 0.921423077583313, 0.31821420788764954, 0.4893713891506195, 0.9123893976211548, 0.4266594350337982, -0.12147898226976395, 0.7226074934005737, 0.1564766764640808, 0.6728712916374207, 1.4229539632797241, 1.084122896194458, -0.5297969579696655, -1.182090401649475, -0.3170645534992218, -1.3205900192260742, -0.32412654161453247, 0.2407790720462799, -0.33549872040748596, -1.7199413776397705, -1.0103235244750977, 1.3355110883712769, 0.4872186481952667, 1.0846225023269653, -2.322153091430664, 0.4549179971218109, 1.772346019744873, 0.946987509727478, 0.5217929482460022, 0.6335688829421997, -0.525960385799408, -1.4891999959945679, -1.007756233215332, -2.119603157043457, 0.6859750151634216, 1.0596649646759033, -0.8001415729522705, 0.5912888050079346, -0.14382866024971008, -0.6975001692771912, -0.6384334564208984, 0.1878044307231903, -1.4767746925354004, -0.6048988699913025, 1.5098035335540771, 0.12904515862464905, 0.8525452017784119, 2.0935418605804443, -0.8928228616714478, -2.425992250442505, -1.0583561658859253, 0.6729717254638672, -1.2179431915283203, -1.2145832777023315, 0.4402826428413391, 0.29132214188575745, 1.4271104335784912, 0.6441137790679932, -0.989183783531189, -1.206992268562317, 0.5297409892082214, 0.05722806230187416, 0.047430843114852905, 1.6907216310501099, 0.35174188017845154, -1.5102424621582031, -2.1236891746520996, 0.08783245086669922, -1.1385228633880615, 0.7585319876670837, -0.20548444986343384, 1.124126672744751, -0.6397293210029602, -1.2776226997375488, 0.8549701571464539, 0.9696335792541504, 0.1319083720445633, -2.8749501705169678, 1.0836666822433472, 0.6209703683853149, 0.5128909945487976, 1.2267835140228271, 0.1636315882205963, 0.7702968120574951, 0.43819060921669006, 0.3361433446407318, 0.22856482863426208, -0.35861074924468994, -0.18165268003940582, 0.18904852867126465, 0.2583090364933014, -0.48124441504478455, 0.6388899683952332, -0.4308488070964813, -0.8409625887870789, 0.4805319607257843, -0.5934497117996216, -1.1198062896728516, 0.2795100510120392, 0.7706491351127625, 1.9746716022491455, -0.7722464799880981, -1.4752545356750488, 0.01247718371450901, -0.5744410753250122, 0.9929096102714539, 0.48881635069847107, 0.725250780582428, 0.760648787021637, 2.041621446609497, 0.667585551738739, -0.19911597669124603, -0.9733036756515503, 0.6322967410087585, 0.0791780948638916, -0.08014778792858124, -1.4399194717407227, 0.871757984161377, 0.6675171852111816, -1.213768720626831], [0.8550791144371033, 0.5180771946907043, 0.7192277312278748, 0.5898086428642273, 0.40996667742729187, 1.638142704963684, 0.023656290024518967, -0.997053325176239, -0.0033361983951181173, 0.06116830185055733, 0.11125422269105911, 0.6825512647628784, -1.5820015668869019, -0.2453567385673523, -1.5025564432144165, -1.8603525161743164, -0.4879707992076874, 2.7825257778167725, 1.969052791595459, 0.19702577590942383, 0.6606078743934631, -0.29424530267715454, -1.2865434885025024, -0.7073313593864441, 1.7867350578308105, -1.2261433601379395, 0.509761393070221, -0.9935070872306824, -0.2783310115337372, -0.6926555037498474, 0.3940926194190979, -0.23478873074054718, -0.2695850729942322, 0.39699023962020874, -1.119091510772705, 1.3593138456344604, -2.4041829109191895, -0.40164440870285034, 1.7967158555984497, 2.4620466232299805, -1.4189800024032593, -2.264822483062744, -1.3646091222763062, 0.7938320636749268, 0.7310726642608643, -1.1147083044052124, 0.01803942583501339, -1.8159559965133667, -1.238873839378357, 0.8206908106803894, 0.9554360508918762, -0.849259614944458, 0.33859413862228394, -1.4980998039245605, 0.849032461643219, 0.7102828621864319, 0.7870960235595703, 0.8971001505851746, -0.465049147605896, -0.8551862239837646, 0.20225927233695984, 1.3498737812042236, -0.47369280457496643, 0.1644047647714615, 0.16545841097831726, -0.613956868648529, 0.29952147603034973, 0.3101590573787689, 2.0938544273376465, 0.10280168801546097, -1.0859564542770386, -1.7579299211502075, -0.8930304050445557, 0.3055427372455597, 0.808648407459259, 0.2562939524650574, 0.5442384481430054, -0.7038183212280273, -0.17678335309028625, 0.5612433552742004, -1.1909161806106567, 0.7126075625419617, 1.8679453134536743, -1.2319605350494385, -0.7957574129104614, -0.587324321269989, -0.47389569878578186, 0.09087377786636353, 1.1386581659317017, -1.0962589979171753, 0.7426939010620117, -0.6455698013305664, 1.5866183042526245, 0.21553553640842438, -0.06900771707296371, 0.1915445625782013, -0.035529833287000656, 1.2751199007034302, 1.1653926372528076, 0.7704582214355469, 0.6264809966087341, 0.9136528968811035, -0.4012506902217865, -1.0490158796310425, 0.2321370244026184, -0.4956956207752228, -0.054083358496427536, 2.2377636432647705, -0.6070720553398132, 0.06850263476371765, 0.10190960019826889, 2.1514394283294678, 0.1346946507692337, 0.73124098777771, -0.5491678714752197, -0.5594888925552368, -0.33083847165107727, 0.2556811273097992, 0.9495495557785034, 1.8889634609222412, 0.9232631325721741, -0.19007614254951477, 2.397296905517578, 0.39809784293174744, -0.3908325433731079, 0.4771828353404999, -0.961730420589447, -0.004122809041291475, 0.4502476751804352, 0.008037885650992393, 0.6592332124710083, 0.7060977816581726, 0.30523377656936646, -0.5684131383895874, 1.606598973274231, 1.100874900817871, -0.37564608454704285, -1.1500818729400635, 1.405776023864746, 0.032065533101558685, -0.2251906543970108, -0.7943011522293091, -0.43817585706710815, -0.11998305469751358, -1.5178827047348022, -0.0187667328864336, -1.3394901752471924, -1.2860832214355469, -0.6544697880744934, -0.22134536504745483, 0.29018330574035645, -0.6846641898155212, -0.7047034502029419, 1.207444667816162, 0.2518826425075531, -0.04886342212557793, 1.2729010581970215, 0.4629533588886261, -0.06010734289884567, -0.680356502532959, -0.9301088452339172, 1.267013669013977, -0.37036049365997314, -1.19541335105896, 0.6681831479072571, -0.3898332118988037, -0.2983464300632477, -0.14392289519309998, 0.30722281336784363, -0.7744749188423157, -0.8721264600753784, 1.7426223754882812, -1.0061345100402832, -0.20211327075958252, -0.39466094970703125, 0.5387468338012695, 0.420457661151886, -1.5381386280059814, -1.0414328575134277, -1.1276166439056396, -0.5308794379234314, -0.6704730987548828, -1.02119779586792, 1.326051950454712, -1.1631145477294922, -0.9021032452583313, 0.9754946231842041, 0.44699153304100037, 3.380936323083006e-05, -0.6260427832603455, 0.8410003781318665, -0.2583106458187103, -0.4796519875526428, -0.23091308772563934, 0.8341577053070068, -0.7165444493293762, -0.07258999347686768, -0.31801649928092957, 0.09129001945257187, -1.0410699844360352, 0.367542564868927, 1.2563987970352173, -1.2189817428588867, 1.5520871877670288, 0.6365640163421631, -1.9883712530136108, -0.56919926404953, -0.5847945809364319, -1.3815945386886597, 1.1959620714187622, -1.0002260208129883, -0.6827911734580994, 2.21614408493042, -0.9157414436340332, 1.140955924987793, 1.338417410850525, 1.1954494714736938, 0.6168022155761719, 0.9987348914146423, -0.7325413823127747, 0.45897573232650757, -0.6912055015563965, -0.12888656556606293, -0.16154520213603973, -0.6708652377128601, -1.1739089488983154, 0.012867940589785576, -0.044671010226011276, -0.3507212996482849, -1.0159873962402344, -1.1222529411315918, -1.6594569683074951, 0.18741822242736816, -2.0110435485839844, 0.8003776669502258, 0.3086731731891632, 0.29188358783721924, -0.019107801839709282, -0.24704323709011078, 1.2080724239349365, -0.4711284339427948, 0.24127653241157532, 0.0704551637172699, -1.0372072458267212, -0.2444215565919876, 1.065932273864746, -0.09702125936746597, -1.495704174041748, -0.3455033004283905, -0.9872230291366577, 0.6986585259437561, 1.0751926898956299, 0.33033841848373413, -1.1527265310287476, 1.349234938621521, 1.0195821523666382, 1.273144245147705, -0.9995702505111694, -0.15020251274108887, 0.0929253101348877, -0.781051754951477, 0.33143240213394165, 1.522838830947876, 1.1188905239105225, -1.1560230255126953, -0.9092089533805847, -0.9870897531509399, 0.008981696330010891, 0.8025866150856018, 0.05622253939509392, 0.2449769526720047, -2.7262232303619385, 0.037258461117744446, 1.5908873081207275, -2.1670918464660645, -0.531644880771637, -1.030081033706665, 2.1289007663726807, -0.4780806601047516, -1.1804159879684448, 0.6774622201919556, -0.05730557069182396, -0.1951010525226593, 0.7641435861587524, 0.42991212010383606, 0.6634321808815002, -0.6097560524940491, 0.7651792168617249, -0.7179630398750305, -0.9782829284667969, 0.04696638137102127, 1.3661545515060425, 0.042745571583509445, 0.6841274499893188, 0.0779331773519516, -0.8609420657157898, 0.11087214201688766, 0.0697900578379631, -1.9089878797531128, 0.6753302216529846, -0.19702540338039398, 0.9846763610839844, -1.5847996473312378, -0.4321736693382263, -0.7310464978218079, 1.6645073890686035, 0.6783195734024048, -1.081286907196045, -1.9027029275894165, 0.5643116235733032, 0.29753321409225464, -1.1400755643844604, 1.5823554992675781, 0.615929901599884, -1.0559486150741577, -0.3411727249622345, 1.5055007934570312, 1.3206720352172852, 1.8173209428787231, -0.6178720593452454, -1.821912169456482, -0.49135610461235046, 0.7896608710289001, -2.5730528831481934, 0.8070772290229797, 0.7248307466506958, -1.2864563465118408, -0.7816494703292847, -0.9323764443397522, -1.5281001329421997, -0.6517373323440552, 0.20656701922416687, 0.3163619935512543, 1.3659331798553467, 0.7907264232635498, 0.39701128005981445, 1.9124053716659546, -0.2115364521741867, -0.07197548449039459, -0.8645699620246887, -1.8223142623901367, -0.18351157009601593, -0.7214723229408264, -0.4973662197589874, 0.6035496592521667, -0.8891204595565796, 0.42422911524772644, 1.1872837543487549, -0.48870766162872314, -0.30390602350234985, 0.25515198707580566, -0.9001245498657227, -1.5964258909225464, 0.3455186188220978, -0.345200777053833, -0.5309555530548096, 0.0688665360212326, -0.9065667390823364, 1.0602284669876099, 0.6344590783119202, 0.8631358742713928, 0.2306063175201416, 0.6369318962097168, -1.8773853778839111, -0.010293038561940193, 0.8786826133728027, -1.6194559335708618, 0.821144163608551, -0.6587522625923157, 0.2608930170536041, 1.142952799797058, 0.6832793354988098, 1.5528732538223267, 0.4282376170158386, -1.1383041143417358, 1.480025291442871, 0.786106288433075, 1.1564456224441528, -0.32565295696258545, 0.444006085395813, 0.4046083688735962, 0.1771007776260376, 0.18453264236450195, -0.1633632779121399, 0.08712778240442276, -0.18765190243721008, 0.5768889784812927, -1.3929928541183472, 0.6487122774124146, -0.6704362034797668, 1.5992976427078247, 0.1133849248290062, -1.6578885316848755, 0.8172741532325745, -1.9331295490264893, -0.46767815947532654, -1.2695622444152832, -0.36716681718826294, 0.3899095058441162, -2.0572893619537354, 2.1711840629577637, -0.5419784188270569, 0.03308578580617905, -0.4659903943538666, 0.0416753813624382, -0.907258927822113, 1.2489635944366455, -0.14928531646728516, 0.17597512900829315, 0.48339784145355225, 0.5840936303138733, -0.1465688794851303, 0.25260061025619507, 0.5106894373893738, -0.7660703063011169, 1.3770099878311157, -0.1395369917154312, 0.9890996813774109, 0.8230242729187012, -0.5040934085845947, -0.455453097820282, -1.0590826272964478, 0.13474668562412262, 1.275734543800354, -0.8880312442779541, 1.4943381547927856, 2.2608470916748047, 0.9661596417427063, 1.2573412656784058, -0.20019382238388062, 0.5681332349777222, -0.3524776101112366, -1.6212000846862793, -0.6149242520332336, 0.1975213587284088, -0.7025536894798279, -1.1867856979370117, -1.674654245376587, 0.7886741161346436, -0.5438776612281799, -0.4276653528213501, -0.57222580909729, -0.5185471177101135, -0.4740901291370392, 0.02216276153922081, 1.515589952468872, -0.14615745842456818, -0.6804347038269043, -0.26897117495536804, 1.505849838256836, 0.6751893162727356, -0.5967418551445007, -1.3183575868606567, -0.6003995537757874, 0.16496887803077698, 0.2606777846813202, 0.7337235808372498, -0.24393172562122345, -3.023024559020996, -1.4824426174163818, -1.7147793769836426, 0.1604224443435669, -0.8744280934333801, 0.46254780888557434, -0.41763338446617126, 0.7797940969467163, -0.4340267777442932, 1.0439900159835815, -0.5170509815216064, -0.7955203652381897, -0.6399407982826233, -0.37543779611587524, -0.3926395773887634, 0.019244683906435966, 1.4294588565826416, -0.4501208961009979, 0.4088195860385895, -0.5193139910697937, 1.0702499151229858, 0.17237530648708344, 0.42198261618614197, -0.9836949706077576, -0.71293705701828, -0.9210084676742554, 1.3561656475067139, 0.019177056849002838, -0.5127692222595215, 0.128538578748703, -0.5855808854103088, 1.2912169694900513, -0.42532894015312195, 1.0279994010925293, -2.8520867824554443, 0.6722082495689392, 0.058674052357673645, 0.0025682891719043255, -0.7314571142196655, 0.06191350519657135, 0.48706185817718506, -0.26988089084625244, -0.913986086845398, 0.05347978696227074, 1.7324100732803345, -1.6032601594924927, -0.7770663499832153, 0.34109175205230713, -0.6461065411567688, -0.5322751402854919, 0.724985659122467, -0.09068670868873596, 0.5082966089248657, 1.1340200901031494, 0.673645555973053, -0.770258903503418, 0.1356702446937561, -0.06009521335363388, 1.2681572437286377, 0.6169748306274414, -0.9237509369850159, -0.8457677960395813, 0.8209576606750488, -1.7198539972305298, -0.0937580093741417, 0.288244366645813, 0.5867272615432739, -0.4476712942123413, 0.2206350713968277, -0.903569221496582, 2.301253318786621, 1.8940131664276123, -0.5311093926429749, -0.412478506565094, -0.3419587314128876, 1.6735100746154785, 1.3690110445022583, -0.5586367249488831, 0.38841670751571655, -1.0382015705108643, 1.364721417427063, -0.9951430559158325, -0.4897129237651825, -0.8463672995567322, -2.4219675064086914, 1.0155853033065796, -1.8316744565963745, 0.6405600905418396, 1.4371544122695923, 0.506432831287384, -0.8176588416099548, -0.591546893119812, 0.07937081158161163, -0.788666307926178, 0.8486375212669373, 0.6005550622940063, 0.8948959708213806, 2.0329957008361816, -0.5730806589126587, 0.576221227645874, -1.2441020011901855, 0.927743136882782, -0.5975036025047302, 0.9675793647766113, 0.2766510844230652, -1.000825047492981, 0.6130481362342834, 0.8513974547386169, 0.7338911294937134, -1.1999282836914062, 1.151725172996521, 0.020381106063723564, -0.14766371250152588, -2.0110509395599365, 0.9578022360801697, 1.2565604448318481, -0.7090612053871155, -3.028879165649414, -1.849448323249817, -1.218205213546753, -0.5778286457061768, 0.7057127356529236, 0.41554975509643555, -0.26113206148147583, 0.202285498380661, 0.9313494563102722, 1.4179131984710693, -1.2876498699188232, -0.16394510865211487, 1.0704096555709839, -0.29911181330680847, 0.20949825644493103, 2.0858116149902344, 0.6802055835723877, -0.06976247578859329, -0.12295344471931458, -1.6967254877090454, 0.31590646505355835, 0.3237723410129547, 0.11570987105369568, -0.3256913125514984, -0.9935314655303955, -0.8515012264251709, -1.1673575639724731, -0.25643429160118103, 1.1067736148834229, 0.41720935702323914, 1.1417334079742432, 1.4000308513641357, 1.1397697925567627, 0.7615953683853149, 0.8644134402275085, 0.3933468163013458, 1.6688098907470703, -0.6317896246910095, -1.0946664810180664, 0.005033043213188648, -1.4548441171646118, 0.38217368721961975, -0.8732466697692871, 0.1647278368473053, 0.5863022208213806, 0.41246965527534485, -1.5575077533721924, -0.2965535819530487, 1.118564248085022, -0.8071566820144653, 0.5746671557426453, 0.7084592580795288, 0.9621491432189941, 2.4165356159210205, 1.8015878200531006, 0.5094539523124695, -1.0811132192611694, -1.5160571336746216, 0.3122163712978363, 1.3374881744384766, 1.0962177515029907, -0.36222904920578003, 0.8109064102172852, -0.8966955542564392, -0.2219853550195694, -0.8190792798995972, -0.07532228529453278, 1.5956273078918457, 1.184998631477356, 0.30186721682548523, 2.138781785964966, -2.5422139167785645, 0.06917808204889297, 0.6987437009811401, 0.24416619539260864, 0.9452384114265442, 1.4686106443405151, -0.9577828049659729, 1.820597529411316, 1.8816734552383423, 0.39570891857147217, 0.6375520825386047, 0.12280820310115814, -1.938293218612671, 1.4754806756973267, -0.06260108947753906, 0.3963377773761749, 0.03713252395391464, -0.20105236768722534, 1.7546753883361816, -1.5127207040786743, 0.08844996243715286, -0.47611531615257263, -1.0654805898666382, -0.4673757255077362, -1.6670411825180054, -0.7076196074485779, -3.5553219318389893, 1.093194842338562, -0.051463935524225235, 1.026316523551941, -2.436990976333618, 0.2750176787376404, -0.9454681873321533, -1.081027865409851, -0.5682142376899719, -4.090629577636719, -0.5614704489707947, 1.0009325742721558, -1.808045744895935, 0.16373935341835022, 0.012476247735321522, 0.07690035551786423, 0.9951238632202148, -1.307578682899475, -0.2508060336112976, -0.24045078456401825, -0.9804953932762146, 0.956585168838501, -0.5027230978012085, -1.4345769882202148, -0.9518665075302124, 0.7527005672454834, 1.2283178567886353, -0.21154004335403442, -1.0583758354187012, 0.6157379150390625, -0.6948281526565552, 0.30659398436546326, -0.5743697881698608, -1.1670838594436646, 0.10213994234800339, 0.21771718561649323, 1.4738527536392212, -0.19582325220108032, 0.24421140551567078, 1.2880220413208008, 0.4373689591884613, -1.693988561630249, -0.012429767288267612, 0.8253543972969055, 0.9142897725105286, -0.0968894511461258, -0.11339017748832703, 0.6924977898597717, 0.9661261439323425, 0.4074147343635559, 0.015190799720585346, 0.7165910005569458, 0.8490041494369507, 0.36038461327552795, -0.8089028596878052, -1.424947738647461, -1.0790218114852905, -1.0642503499984741, 0.5150089263916016, -0.7044609189033508, -0.012447391636669636, -0.36878496408462524, 0.4800189137458801, -0.4640263617038727, -0.26672589778900146, -0.8682965636253357, -0.04772370308637619, -2.3470675945281982, 1.1392885446548462, -0.9736470580101013, 1.2336052656173706, -0.20609521865844727, -1.1692790985107422, 0.5699988603591919, -0.05323972553014755, 0.7543060183525085, -1.01386296749115, 0.6632670164108276, -0.17880626022815704, 0.19931533932685852, 1.3061575889587402, -0.7463898658752441, 0.8901119828224182, 0.15952570736408234, -1.2408262491226196, 0.41778698563575745, -0.02747795172035694, -0.3232818841934204, 1.5674256086349487, 0.5750910639762878, 0.618617594242096, 0.7024793028831482, -1.146783471107483, -0.23791028559207916, 0.22698044776916504, -0.18776346743106842, 0.4159364700317383, -1.2957849502563477, -1.6883955001831055, 0.2635527551174164, 1.4285482168197632, -0.46993914246559143, 1.2147332429885864, 0.2922282814979553, 0.15272444486618042, 0.1714884638786316, -1.5825823545455933, 0.1357462853193283, 0.28892993927001953, 0.1787346750497818, 0.14259730279445648, 0.1966787725687027, -1.0201961994171143, -0.05732935294508934, -0.3067916929721832, -0.7486572265625, -0.40859082341194153, -1.142649531364441, 0.4195493459701538, -1.4808084964752197, 1.7255241870880127, -2.249178409576416, 0.11066870391368866, -2.3474433422088623, -1.0978984832763672, 0.9324454069137573, 0.6908621788024902, 0.07201461493968964, 0.09946230053901672, 1.5311750173568726, -1.50078547000885, -1.16372811794281, 0.038254156708717346, -1.0386546850204468, -1.1010295152664185, 0.18190819025039673, -1.4867061376571655, -1.204642415046692, -0.07858359813690186, -0.452389657497406, -0.10938291996717453, 0.7330527305603027, 0.041598498821258545, 1.686939001083374, -0.056540023535490036, -0.13323737680912018, -0.6312988996505737, 1.3737391233444214, -1.10495924949646, 1.6639854907989502, -1.393556833267212, 1.7452746629714966, -0.06691677868366241, 1.4552432298660278, 0.31097152829170227, -0.9023234248161316, 0.4066239297389984, 0.27452975511550903, -0.2473578304052353, -0.13831497728824615, 0.9427881240844727, 0.555600106716156, 0.43608495593070984, 0.07453764230012894, 0.4401422441005707, 1.2396472692489624, -1.6250413656234741, 1.2454321384429932, 0.13740932941436768, -1.8447216749191284, -1.1393156051635742, 0.2146313339471817, -0.06385936588048935, 0.6227614283561707, 0.6163862943649292, -0.5527568459510803, 0.34522804617881775, 0.5486894249916077, -0.24122408032417297, -0.9758908748626709, -1.2967603206634521, -0.4606425166130066, -1.2561938762664795, 1.4700907468795776, -0.08798401802778244, 0.1387455314397812, -1.1231459379196167, 0.025361988693475723, 0.07445640116930008, -0.3661513030529022, 0.39947983622550964, 1.1097904443740845, 0.5832454562187195, 0.9714025855064392, -0.7654725909233093, -0.0855465680360794, 0.4208226799964905, 0.6949177980422974, 1.9749557971954346, -1.5133569240570068, 1.888715386390686, 0.13061745464801788, 1.366358757019043, 0.061260249465703964, 1.2405728101730347, 1.8810477256774902, 1.0602912902832031, 1.544554591178894, -1.0914301872253418, -0.2716705799102783, 0.5928879976272583, -1.1357537508010864, -1.302231788635254, -0.03695319965481758, -0.46081045269966125, 0.3086089491844177, 0.745236873626709, 1.0326050519943237, 0.6462911367416382, 0.3928977847099304, 1.0617635250091553, -0.04926395043730736, 0.4672134816646576, -0.8848032355308533, 0.4797036945819855, -1.4183355569839478, -1.0772452354431152, -1.3096195459365845, -1.159912109375, -0.3613375127315521, 0.13895933330059052, -0.1801481992006302, -0.7293339371681213, -0.6300424933433533, 0.6851984858512878, -1.067588448524475, -0.1567251980304718, 0.3340807855129242, -0.6078070402145386, -1.8553894758224487, 0.4278928339481354, 2.83180570602417, 1.7286251783370972, 0.11660850048065186, -0.3286910951137543, 0.1714227795600891, -0.5625713467597961, 1.4350497722625732, -1.029896855354309, -1.0686193704605103, 0.044558994472026825, -1.0164388418197632, -0.16843892633914948, -0.17335636913776398, 1.4964436292648315, -0.1190071552991867, -0.07109185308218002, 0.3129001557826996, 1.7715470790863037, 0.19197122752666473, 0.46314650774002075, -2.1203317642211914, -1.9268428087234497, 1.3832528591156006, 1.210453748703003, 0.5529097318649292, -0.05553150177001953, -2.129066228866577, 1.242658019065857, -0.16864055395126343, 0.19031581282615662, 0.2426796704530716, 1.1217576265335083, 1.1438063383102417, 1.0778405666351318, 0.6169789433479309, 0.41279739141464233, 0.809299647808075, 0.9536020159721375, 0.9679274559020996, 0.8579237461090088, -0.1939481943845749, 0.7310107946395874, 0.0073518757708370686, 0.1978309601545334, 0.23798684775829315, -0.32604971528053284, 0.5106472373008728, 1.6277599334716797, -1.4777668714523315, -0.03929677605628967, 1.4136000871658325, -0.6660786271095276, -1.4100871086120605, 1.3776764869689941, 1.1828581094741821, -1.4347891807556152, 0.32861387729644775, -1.527969479560852, 1.7135165929794312, 0.6142118573188782, -1.543400764465332, 0.3572408854961395, 0.2188671976327896, 1.1602998971939087, 0.748911440372467, -0.7230875492095947, 1.3645310401916504, 0.5542522072792053, 0.5508168339729309, -0.5882940888404846, -1.0893381834030151, 0.40797778964042664, 0.9260250329971313, -0.3361809253692627, 1.0566068887710571, 2.3461427688598633, 0.09057366102933884, -2.286440372467041, 0.9275028705596924, 0.9884145259857178, -0.12791530787944794, 2.3678903579711914, 1.3750182390213013, -0.31805914640426636, -0.6039337515830994, 1.2589845657348633, -0.21157921850681305, 2.038865327835083, 0.7024850249290466, 1.1661392450332642, -0.48297569155693054, -0.984316349029541, 0.529869794845581, -0.3583255708217621, 0.6748507022857666, -0.3272189199924469, 0.4756280481815338, -0.8605666756629944, 0.5412857532501221, 0.6849722266197205, -0.3989921808242798, 0.3547652065753937, 1.7779724597930908, 0.8935125470161438, 1.6338156461715698, 1.2596038579940796, 0.3852386772632599, 1.3438173532485962, 1.0574299097061157, -0.4330355226993561, -0.408586323261261], [0.23454461991786957, -0.3009953498840332, 0.369878351688385, -0.031105775386095047, -0.7697073817253113, -0.33337441086769104, -1.7006336450576782, 1.023241639137268, -1.0763574838638306, 1.6252670288085938, 0.8311176896095276, 0.04213075339794159, 1.5851826667785645, -0.09522145986557007, 0.45760613679885864, -0.0475909486413002, 0.12938429415225983, 0.826195478439331, -0.1502087563276291, 0.4770676791667938, -0.03628871217370033, 0.7193818092346191, 0.09239166229963303, -0.7913634777069092, 0.47026050090789795, -0.40550804138183594, -0.8112263083457947, 0.6407349109649658, -0.4858357310295105, -1.950271487236023, -0.1668495535850525, -0.8046833276748657, 0.8095076084136963, 1.1142523288726807, -0.2926434576511383, 0.7036510109901428, -0.14867225289344788, 0.1231905147433281, 1.7636417150497437, -0.7527365684509277, 2.32523775100708, 0.6621710658073425, 1.745605707168579, -0.6973103880882263, 0.620301365852356, 0.42830199003219604, -0.6069404482841492, 1.7051329612731934, 0.4633447229862213, 0.007478478364646435, -0.602972149848938, 0.18493156135082245, 0.7448014616966248, 0.7237635850906372, -0.17402252554893494, 1.8774994611740112, -0.7964390516281128, -0.745627224445343, -1.2321406602859497, -1.3038781881332397, 0.5371649265289307, 1.144419550895691, -1.008248209953308, 0.25085243582725525, -0.9428809881210327, -0.35731858015060425, -0.4294455945491791, 0.6298432946205139, -0.6566309332847595, -0.5133465528488159, -0.34890028834342957, 0.7776490449905396, -0.36213311553001404, 0.16483670473098755, -1.1399470567703247, -0.39306455850601196, -0.881544828414917, 0.2852492332458496, -1.0360859632492065, -0.9180264472961426, 0.3108481764793396, -1.5954465866088867, 1.3012001514434814, 0.5385915637016296, -0.5813132524490356, -0.9130794405937195, 1.1205852031707764, 0.06488396227359772, -0.575829029083252, -0.20941275358200073, 1.0517551898956299, 0.46698424220085144, -0.2741398513317108, -0.9538335800170898, 0.19294141232967377, 0.15746687352657318, 0.43511277437210083, 0.4405558407306671, 0.9137914776802063, -0.5131794214248657, 0.6828658580780029, 0.09447797387838364, 0.043508727103471756, 0.527152419090271, -1.4434833526611328, 1.8684278726577759, 1.3917149305343628, -1.6330288648605347, 0.1327783763408661, 0.45808660984039307, -0.9524694681167603, -0.55082106590271, 0.3552674353122711, -0.9598482251167297, 0.14113138616085052, 2.1126556396484375, 0.5234189033508301, -0.10217943042516708, -0.985096275806427, -1.2981833219528198, -0.9201436042785645, 1.1211615800857544, -2.916748523712158, 0.889840304851532, 0.12554238736629486, -0.7062571048736572, 2.0564796924591064, 0.7670882344245911, 1.2878330945968628, -0.2260727882385254, -0.7182905673980713, -0.2725163400173187, 1.0534826517105103, -0.4413680136203766, 0.3450264036655426, 1.4966667890548706, -1.1166237592697144, -0.24089056253433228, 0.37455859780311584, 1.4702732563018799, 0.9322939515113831, 0.14132939279079437, 0.9179645776748657, 0.5412449836730957, 0.45491406321525574, 0.20018161833286285, 0.6323543190956116, -0.8865629434585571, 2.269458055496216, 0.4365268051624298, -0.9614515900611877, -1.1211882829666138, -1.382044792175293, 2.3028147220611572, -1.149538278579712, 0.8558436632156372, -0.24239985644817352, -1.8873305320739746, -0.30711713433265686, -0.9691771864891052, -2.2390835285186768, -0.33711063861846924, 1.2311820983886719, 0.6486986875534058, -0.8543949127197266, -1.3906830549240112, -1.0160471200942993, -1.0280683040618896, -0.6669471263885498, -1.2327895164489746, 1.6447736024856567, -0.0459720753133297, -0.45322564244270325, -0.3606700599193573, 1.1806271076202393, -0.0520855151116848, -0.956540048122406, -0.8263919949531555, -1.0607969760894775, 0.5476382970809937, 0.6309272646903992, -2.7184200286865234, 0.4906730651855469, 1.7571762800216675, -1.7394386529922485, 0.2448563128709793, 0.8696120977401733, 0.6213864088058472, 1.3682430982589722, 0.19281721115112305, -0.39509186148643494, -0.22804167866706848, 1.5746517181396484, 0.04981251806020737, -0.33629441261291504, 1.884656310081482, -0.3869853913784027, -0.029657520353794098, -2.05362606048584, 0.3851125240325928, 1.715867042541504, -0.7692191004753113, -0.776759684085846, -2.352856159210205, -0.771024227142334, -0.50212562084198, -0.17789976298809052, -0.7897601127624512, -1.4868595600128174, -0.317059189081192, -0.184348925948143, 1.4005976915359497, 0.8700687289237976, -0.7505741715431213, -0.6626383066177368, 0.06835462898015976, -0.7308104634284973, -0.026063332334160805, 0.09030833095312119, -1.8016777038574219, -1.126135230064392, -0.5732392072677612, 1.1400755643844604, 2.2262918949127197, -1.0286331176757812, -0.6845438480377197, -0.23005378246307373, 1.770301342010498, -1.1129579544067383, -0.6523492336273193, 0.49260836839675903, 0.6019088625907898, -0.3936864137649536, 2.194706439971924, 0.5501404404640198, -0.30153483152389526, -0.5217157006263733, 1.4378520250320435, 1.810344934463501, -0.2768901586532593, -0.23349373042583466, 0.3538459241390228, -0.9681732058525085, -0.5026177167892456, 1.1144523620605469, -1.592108964920044, -0.34249481558799744, -0.0305681973695755, -0.8162024021148682, -0.9746667742729187, 0.26391860842704773, -0.38684871792793274, 0.651395857334137, -0.17300857603549957, -1.3524377346038818, -0.431073933839798, -1.6436203718185425, -0.5885970592498779, -1.2810986042022705, -0.0029909375589340925, -1.8778027296066284, -0.13977903127670288, -0.2149365395307541, -0.7563299536705017, -0.8425297141075134, -0.485378235578537, -0.7343189716339111, 0.20161131024360657, 0.39085227251052856, -0.9855514168739319, -1.175166130065918, 0.42656800150871277, -0.2633410096168518, 0.14508791267871857, -1.432678461074829, -0.4539453685283661, 0.2696411907672882, 0.08068932592868805, 0.29782193899154663, -0.29199284315109253, -0.05313783511519432, 0.4270308017730713, 1.6800934076309204, 0.733122706413269, 0.3511860966682434, 0.43963587284088135, 1.958706259727478, 1.7665486335754395, -0.3139384090900421, 0.31116896867752075, 0.25885188579559326, -0.40794092416763306, -0.39064112305641174, -0.3650636374950409, 1.62947678565979, 1.1631836891174316, -0.6672976016998291, 0.6688607335090637, -1.5063135623931885, -0.48653683066368103, -0.7020401954650879, 0.8158218264579773, 0.7407580018043518, -0.041339848190546036, 0.1887427717447281, -0.022746004164218903, 0.8307190537452698, -1.3435243368148804, -1.2363617420196533, -0.6657578349113464, 0.8675602078437805, -1.5154160261154175, -1.6100064516067505, 0.16542771458625793, -1.1580326557159424, 0.867097020149231, 0.48965489864349365, 0.44848814606666565, 0.7468721270561218, 0.25447922945022583, 1.114957571029663, 0.03513766825199127, -0.4799778163433075, -2.0548949241638184, 1.8193206787109375, 2.1564395427703857, -0.8377134799957275, -0.4482875466346741, -0.8754392862319946, -0.8706817626953125, -1.2913936376571655, -0.3368103802204132, -0.27149710059165955, 1.494956612586975, -0.8293925523757935, -0.241710364818573, -0.0011678963201120496, 1.0455325841903687, -1.813058853149414, 1.189699649810791, 0.11046392470598221, -0.14284934103488922, -0.9383968114852905, -0.048423536121845245, -0.8022152185440063, -1.6208124160766602, 0.15652379393577576, -1.0684418678283691, -0.4199954569339752, -1.1808216571807861, -0.058623723685741425, -0.8934430480003357, -0.3160957098007202, 0.09549187868833542, -0.44345876574516296, -0.6825904846191406, -0.29660964012145996, -0.2617737352848053, 0.05885583534836769, -1.4330743551254272, -0.049001503735780716, 1.016625165939331, 1.7234838008880615, -0.6204622983932495, 0.2990897297859192, -1.1974941492080688, -0.4110278785228729, -0.1400412917137146, 0.7194033265113831, 1.0606545209884644, -0.06445744633674622, -1.0190459489822388, -0.7902609705924988, 1.4902071952819824, -0.4406895339488983, 0.8345963954925537, -0.09603331983089447, -0.5529835820198059, 1.1040523052215576, 1.107305884361267, 0.26268479228019714, 1.135209560394287, -0.610476553440094, -0.36931487917900085, -1.8101192712783813, -1.4523118734359741, -1.828060269355774, -0.9790618419647217, 0.19636717438697815, 0.3783554434776306, 0.09925501048564911, -0.39659011363983154, 0.2263265997171402, 1.7652461528778076, 1.0507992506027222, 0.12015601992607117, -0.8113315105438232, 0.8640493154525757, 2.654090166091919, 0.9800370931625366, 0.18587838113307953, -1.272752285003662, 0.23774994909763336, -0.33419257402420044, -0.9232143759727478, 1.2306749820709229, 1.7266491651535034, -0.7198618054389954, 1.467759132385254, 0.6512934565544128, 0.5614070892333984, -1.4820483922958374, 0.5802215337753296, -0.973182737827301, 0.9319658279418945, -2.3935487270355225, -0.07852211594581604, 0.18107588589191437, -0.3652212619781494, -0.335580974817276, 0.5919569730758667, 0.08166392147541046, 1.0682345628738403, -0.408099889755249, -1.3844459056854248, -1.3833017349243164, 0.8875282406806946, 3.014420747756958, -0.3468003273010254, -2.041212797164917, -0.7013235688209534, 0.05336996540427208, 0.20620675384998322, 0.567724347114563, 0.9686707258224487, -0.5976447463035583, -0.05342722311615944, -0.4834487736225128, 0.9367852210998535, -0.6223909258842468, 1.173224925994873, 1.3742399215698242, -1.7233519554138184, -1.296711802482605, 0.9265565276145935, 0.007978344336152077, -0.6878176927566528, 0.3957531452178955, 3.92978145100642e-05, 1.451481580734253, -0.2807150185108185, 1.3137186765670776, -1.2601284980773926, 0.07929167151451111, -0.5948370099067688, 0.09096452593803406, -1.6664890050888062, 0.2791089713573456, -0.3215332627296448, 0.23714695870876312, -0.17124739289283752, 1.0046448707580566, -0.28225570917129517, 0.3974232077598572, 0.909267008304596, 1.3082319498062134, 1.7100979089736938, 1.2226210832595825, -0.07891100645065308, 1.1566457748413086, 0.08559435606002808, -1.1122715473175049, -0.10025949031114578, -0.3331528902053833, -0.15148642659187317, 0.09611061960458755, -0.5132210850715637, -1.5353343486785889, -1.6826307773590088, -1.1972910165786743, -1.2531776428222656, -0.6720120906829834, 0.19236023724079132, 0.8613972067832947, -0.7228835821151733, -0.52498459815979, 1.299788475036621, 0.04492689296603203, -0.7153446674346924, 0.3307074308395386, 0.3274069130420685, -1.0816611051559448, 0.15610060095787048, -2.5842819213867188, -0.1584717035293579, 0.1569942831993103, -1.4027384519577026, 0.5984092950820923, -0.6678556799888611, -0.9394245147705078, 1.0282459259033203, -0.20155960321426392, 1.3472810983657837, -0.39183515310287476, 0.04596661776304245, 0.407683789730072, -0.8878960013389587, 0.6961882710456848, -1.4850209951400757, 0.9562558531761169, -0.19943027198314667, 1.4965438842773438, 1.0022666454315186, 0.024528101086616516, -1.6291465759277344, -0.5362913012504578, 0.630252480506897, -0.08534099161624908, -0.028181523084640503, -0.10588214546442032, -3.118568181991577, 0.2310992032289505, -1.8033844232559204, 1.2067124843597412, -0.19913233816623688, 1.6574214696884155, 0.15694782137870789, 1.485863208770752, -0.2624281942844391, -0.2002471685409546, -1.0417910814285278, 0.9034900665283203, -0.20833535492420197, 0.3501270115375519, -0.3983099162578583, 0.30037209391593933, 0.1469200700521469, 0.2421223521232605, 0.8839395046234131, -1.03877592086792, 1.2004212141036987, 0.418534517288208, -1.5499975681304932, 1.4453777074813843, -1.6828763484954834, 1.2579023838043213, -0.10126852989196777, -0.45134255290031433, -1.1046147346496582, -0.44898298382759094, 2.1171650886535645, 0.8650211095809937, -1.8542556762695312, 0.29584670066833496, 0.25017061829566956, -0.6536362767219543, -1.9032244682312012, -0.9355636835098267, 0.5982276201248169, 1.6128289699554443, 0.4323936700820923, 1.2886297702789307, 0.2906111180782318, -0.6430507898330688, -1.93069326877594, -1.291163444519043, 0.9276584982872009, 0.911499559879303, -0.010844209231436253, 0.6330575346946716, 1.0812147855758667, -1.39816153049469, -1.47708261013031, 0.19711796939373016, -0.45330893993377686, 0.7711731195449829, 2.8348453044891357, -1.8660905361175537, 0.1807439625263214, -0.5952930450439453, 0.6250055432319641, 1.229171633720398, -1.4226651191711426, -1.4760621786117554, 0.2788065969944, -0.6991790533065796, 0.4549713730812073, 0.6096555590629578, 0.16921639442443848, 1.475870966911316, 0.5550616979598999, 1.0929789543151855, -0.32730457186698914, -0.08362230658531189, 0.9715139865875244, -1.2267265319824219, 1.0304962396621704, 0.43568429350852966, 0.917269766330719, -0.3630468547344208, 0.18033769726753235, 1.7145226001739502, -0.8168997764587402, -0.9794567227363586, 0.7111824154853821, 0.3964650332927704, -0.5183373689651489, -0.7267695069313049, 1.2685720920562744, -0.33854401111602783, -0.28185197710990906, 0.3082171380519867, -1.1269242763519287, 0.7553992867469788, -0.9240253567695618, -1.3822396993637085, -0.5821141600608826, 0.8863727450370789, 1.0845974683761597, 0.4713001251220703, 0.5225287079811096, 0.10211947560310364, 0.996910810470581, 0.4830894470214844, 0.548769474029541, 0.027471601963043213, 0.21440225839614868, 2.48565673828125, 0.9715015292167664, -1.1902046203613281, -0.4356389343738556, -0.04827062785625458, -1.4298112392425537, -0.8231322765350342, 0.9905825257301331, -2.0716733932495117, -0.6197219491004944, 0.15994112193584442, 0.9950403571128845, -0.8379361033439636, 0.07311881333589554, 1.2564802169799805, 0.27582162618637085, 0.8035510778427124, -1.3581472635269165, 0.5803167819976807, -1.2970764636993408, 0.3809174597263336, 0.9050984978675842, 1.5521613359451294, 0.34237682819366455, 1.0281710624694824, 0.24888837337493896, -0.9885870814323425, 1.445815086364746, -0.7006801962852478, 2.3411009311676025, -0.39002126455307007, 0.2046065479516983, 1.0130411386489868, 0.08088713139295578, -0.3930995464324951, -1.3506070375442505, -0.21036311984062195, -0.27087336778640747, -1.6816498041152954, 1.4193873405456543, -0.2539843022823334, 0.7073628902435303, 0.2897309958934784, 0.28657135367393494, 0.42277300357818604, -0.13096444308757782, -1.1100808382034302, 0.8335815668106079, -1.0593494176864624, 0.5531941056251526, -0.8179259896278381, 1.2302244901657104, -1.675750732421875, 0.7394058704376221, -1.1841285228729248, -0.868808925151825, 0.003135192906484008, 0.13659484684467316, -0.33151495456695557, -0.7361491918563843, -0.0074343434534966946, 0.6892465949058533, -0.31234604120254517, -0.6279740929603577, -1.9734539985656738, 0.430563747882843, 0.1536312997341156, -0.821542501449585, 1.7226324081420898, -0.7672473788261414, 0.34084880352020264, 1.368894338607788, -0.7292128205299377, -0.39496031403541565, -2.6332011222839355, 1.4990428686141968, 0.6016995310783386, 0.9354493618011475, 1.1129570007324219, 0.21429674327373505, 0.4924749433994293, -1.4962047338485718, 1.5070490837097168, -0.09216140955686569, -0.44977006316185, -0.37237030267715454, -0.1990227997303009, 1.2260377407073975, -1.8262367248535156, -1.494507074356079, -1.7560425996780396, 2.0792179107666016, -0.3821616470813751, 0.08500344306230545, 1.1539337635040283, -0.386083722114563, 1.0020933151245117, 0.9030225872993469, -0.35795527696609497, 0.5800520777702332, 0.7113933563232422, -1.3431814908981323, 0.316079318523407, 1.1354867219924927, 0.4701511561870575, 1.588865876197815, -0.18012167513370514, 1.8208884000778198, 0.9410011172294617, 1.525303602218628, 0.0005365474498830736, -0.8896763324737549, 0.800930380821228, 0.07307448238134384, -0.38763198256492615, 0.9198383688926697, -0.023320330306887627, 0.4850376844406128, 0.8925603628158569, -0.28136739134788513, -0.05928831174969673, -1.1011323928833008, 1.054607629776001, 0.9077354669570923, -0.025806566700339317, -1.204390525817871, 0.5123713612556458, -0.3225962519645691, -0.47841107845306396, -0.687866747379303, 0.643136203289032, 0.3541426658630371, -0.8908853530883789, -0.15066243708133698, 0.27708983421325684, 1.052161455154419, -1.2593353986740112, -0.53780597448349, -0.737968385219574, 0.8070175647735596, 0.44393450021743774, 1.8358793258666992, -2.3977088928222656, 0.8319359421730042, 0.8954002261161804, 0.19583861529827118, 1.0951963663101196, 0.28080642223358154, -1.0410964488983154, -1.5822540521621704, -0.4941476881504059, -0.1436399668455124, -0.021049823611974716, 1.3973140716552734, 1.8458678722381592, 2.3294119834899902, -0.6056265234947205, -0.7369398474693298, -0.8201545476913452, -1.634569764137268, 0.6189649105072021, -0.5290668606758118, 0.5889419317245483, -0.18566258251667023, -1.9687778949737549, 1.668571949005127, -1.1315553188323975, -1.0698459148406982, 1.1111913919448853, -1.049797773361206, 0.9179976582527161, 1.0540434122085571, 0.37906622886657715, -0.47438138723373413, 1.053985357284546, -0.5252732634544373, 1.2772186994552612, -1.7953544855117798, -0.9537920355796814, -0.06579197943210602, -0.06646372377872467, 0.06150372698903084, 0.4627057909965515, 0.18639931082725525, 0.6909141540527344, 1.9396171569824219, 0.18891242146492004, 0.11333439499139786, 0.21778959035873413, -0.5094600915908813, -0.06388744711875916, 0.5429007411003113, -1.0377259254455566, 0.10336753726005554, 0.24161756038665771, -1.0963069200515747, 0.313387006521225, 0.6098975539207458, -1.0352277755737305, 0.015466206707060337, 0.4968335032463074, -0.6574971079826355, 0.8854024410247803, 0.7279738187789917, -0.25014835596084595, 0.14678196609020233, -0.2622232139110565, 0.6040031909942627, -1.182651400566101, 1.7901690006256104, 0.976385235786438, -1.4590895175933838, -0.3122095465660095, 0.5823646783828735, -1.0902414321899414, -0.9321151971817017, -0.7797256708145142, 0.23829568922519684, -0.8022032976150513, 0.0038722623139619827, 0.3110099136829376, 0.9414080381393433, 1.2680084705352783, -0.341341108083725, -2.5130298137664795, 0.9588465690612793, -0.42223143577575684, -0.7092301249504089, 0.22996702790260315, 0.9733705520629883, 0.941606879234314, -1.2154486179351807, -0.43734025955200195, -1.1467407941818237, -1.7916496992111206, 1.2203187942504883, -1.059596300125122, 0.28063997626304626, -0.8999190330505371, -0.07515542209148407, 1.3292521238327026, -0.7566282153129578, 0.2912621796131134, -0.1353542059659958, 0.6478158235549927, -1.9753196239471436, 1.3717865943908691, -0.9730805158615112, -0.7541862726211548, -1.1508201360702515, -2.3935141563415527, -0.2901841104030609, 0.4555142819881439, -1.585854411125183, 0.9205660820007324, 0.23480433225631714, -0.06652931869029999, -0.24453985691070557, 0.7969529032707214, -0.7834725379943848, -0.4220395088195801, -0.781524658203125, 0.6738321781158447, 0.540156900882721, -1.610241174697876, 0.3006812334060669, 1.4882508516311646, 1.1139779090881348, -0.6280440092086792, 0.9095330238342285, 1.2280384302139282, -1.6414357423782349, -0.18788771331310272, 0.7592262625694275, 0.7103692889213562, -0.4288714826107025, 0.19180187582969666, -0.4178835153579712, 1.765453815460205, 0.1646519899368286, 1.1231486797332764, 0.0412633903324604, 2.393454074859619, -0.2604384124279022, -0.6504435539245605, 1.269352912902832, 0.6407690048217773, -0.7204782366752625, 1.0023175477981567, 0.6531960964202881, 0.2857820391654968, 0.2295679748058319, 0.08766361325979233, -0.4144638180732727, -0.0548989400267601, 0.9841311573982239, -0.25906527042388916, -0.6252183318138123, -1.2732170820236206, -0.3993685245513916, 1.345011591911316, 1.385453224182129, -0.9934793710708618, -0.835725724697113, 0.6778262257575989, -1.1459412574768066, 0.10042840987443924, -1.3718106746673584, 0.09279234707355499, 0.7453731894493103, 1.6292431354522705, -0.7024214267730713, -0.35137245059013367, -0.32962265610694885, 0.4159662425518036, 0.25427812337875366, -0.680537223815918, 3.0901618003845215, -2.145003318786621, -1.679398536682129, 1.4948654174804688, 1.1759448051452637, -1.5852603912353516, 0.049624864012002945, 0.9992637038230896, -0.8808287382125854, -0.20960280299186707, 0.12747539579868317, 1.7209807634353638, -0.0014215538976714015, 0.463633269071579, -0.8765343427658081, -0.5690748691558838, 0.47287416458129883, 1.8333263397216797, 0.3799712359905243, -0.2625839114189148, -3.097290515899658, -1.6721916198730469, -0.09153362363576889, -0.36853155493736267, 0.6507977843284607, 0.7856941819190979, -0.6415054202079773, 2.2796664237976074, -1.4802223443984985, 1.3610632419586182, -0.07314345985651016, -1.7477960586547852, -1.141979455947876, -0.3828522562980652, -0.07313945144414902, -0.4988766014575958, 1.3812050819396973, 0.08717264980077744, 0.7011305093765259, 0.5733993649482727, 1.114692211151123, -1.2999969720840454, -0.8906104564666748, 2.0677809715270996, -0.40468791127204895, -0.33176949620246887, -2.170201301574707, -1.1852291822433472, 1.3171885013580322, 1.0357011556625366, -0.4674627482891083, 0.08985242247581482, 1.3669922351837158, -0.7265154719352722, 1.8992996215820312, 1.3789827823638916, 0.04533030092716217, 0.023076461628079414, 0.21981795132160187, -0.4533271789550781, 0.09191395342350006, 0.42988815903663635, -1.3135031461715698, 2.280512571334839, 0.9872600436210632, -0.1756511926651001, -0.3667660057544708, 0.33320239186286926, -1.524504542350769, -0.7752846479415894, 0.6337032914161682, -1.0006572008132935, 0.9365436434745789, 0.8586385846138, -1.2156556844711304, -0.6470649242401123, 0.1502973735332489, 1.079039454460144, -0.8828290700912476, -0.5736967325210571], [0.8485998511314392, 0.4877648651599884, -0.0270407535135746, -0.7654022574424744, 0.6302852630615234, -1.2209246158599854, 1.7757455110549927, 0.908883273601532, -2.048800468444824, 0.44547194242477417, 0.6355034708976746, 0.08658406883478165, 0.27829331159591675, 1.1122772693634033, -0.8928652405738831, 0.9065423011779785, -1.2362734079360962, 0.6616691946983337, 0.1542660892009735, 0.168256476521492, 1.5780396461486816, 0.32194212079048157, -0.8479309678077698, -0.8409674167633057, -0.6703073382377625, -0.8306106328964233, 0.5655254125595093, 1.6994341611862183, 0.5161672234535217, -2.2791168689727783, 0.11811614781618118, -0.33927977085113525, -0.12659098207950592, 1.6305406093597412, 0.4637317657470703, 0.8293341994285583, 1.3902498483657837, -0.6644966006278992, 0.5563622713088989, -1.0220210552215576, -1.468648910522461, 0.03489871695637703, -1.4263553619384766, -2.050917863845825, 0.6761064529418945, -1.6339772939682007, 0.7133415341377258, -1.7309398651123047, -0.4177691340446472, 0.458199679851532, -2.0805094242095947, -1.3101065158843994, -0.9187254905700684, 0.1024061068892479, -0.5145719051361084, -1.0458556413650513, -0.11081767082214355, -0.8164815306663513, -0.23784196376800537, 0.21275022625923157, 0.6498264670372009, 0.5997441411018372, -1.2305974960327148, 0.486538290977478, -0.5821772813796997, 0.5991402864456177, -0.6454638242721558, -0.3397916257381439, -1.2094135284423828, 0.28025028109550476, -0.15061065554618835, -0.4969484210014343, 1.0532350540161133, 0.26437607407569885, -0.09703074395656586, 0.579071581363678, 0.5957767963409424, 1.1463624238967896, -0.7632072567939758, 1.3986464738845825, 0.8603787422180176, -2.0533783435821533, -0.10631630569696426, -0.6490830779075623, 1.0210464000701904, -1.7556710243225098, -0.7407753467559814, -2.0909817218780518, 0.24186865985393524, 1.0457673072814941, -2.5844690799713135, -1.1124666929244995, 0.01860981062054634, 0.04230382293462753, -0.37604209780693054, 0.06869564950466156, -0.5358153581619263, 0.28889331221580505, -2.2778165340423584, 0.09275532513856888, 0.12432219833135605, -1.8790442943572998, 1.0788333415985107, -0.9028552174568176, 0.05549737066030502, 0.0009715023916214705, 0.43049320578575134, 0.3514752984046936, 1.761334776878357, -0.031184237450361252, 0.019679078832268715, 1.1957350969314575, 1.0928997993469238, -0.3606534004211426, 0.5801815390586853, 1.666900396347046, 0.4796242415904999, 0.20253905653953552, -0.7109017372131348, 0.9121473431587219, 0.13125987350940704, -0.16021306812763214, 0.4031798541545868, 0.7041166424751282, 0.01113332062959671, 1.614209771156311, -0.729878306388855, -1.336405634880066, -0.3464926779270172, 1.0458998680114746, 1.1716121435165405, 0.42309892177581787, -2.0499086380004883, -0.7037850022315979, -0.7357756495475769, 0.9416273236274719, -1.0042890310287476, -0.25628870725631714, -0.22124090790748596, 0.9655975699424744, 0.5361975431442261, -0.4550604820251465, -1.4523708820343018, 2.9326555728912354, 1.8614528179168701, 2.463226556777954, 0.496988445520401, -0.7077186703681946, -0.20472580194473267, -0.27507510781288147, 0.8185283541679382, 0.558121919631958, 1.2858158349990845, 1.338005781173706, 0.8757417798042297, 0.5113340616226196, -0.154745414853096, -0.6898892521858215, 1.2100880146026611, -2.3734076023101807, 1.0677251815795898, 1.9263219833374023, 1.0459558963775635, -1.1690410375595093, 0.17355293035507202, -0.13300025463104248, -0.34920260310173035, 1.039586067199707, -0.07540886849164963, -2.016692638397217, 0.6304376125335693, -0.11498671770095825, 0.623598039150238, 1.4501475095748901, -0.0825430229306221, 1.4127634763717651, -0.5333964228630066, 0.6458600759506226, 1.681700348854065, -1.2414277791976929, -1.4948437213897705, 0.012446297332644463, 0.0011489612516015768, -0.7118086218833923, -0.04347455874085426, 0.13493451476097107, 0.5913418531417847, -0.2118891030550003, 1.5594758987426758, 1.5494163036346436, -1.6229885816574097, -0.5984418392181396, -0.17676781117916107, 0.6677281260490417, 0.29406797885894775, 0.635805070400238, 0.35407915711402893, -0.21531108021736145, -0.21609048545360565, -0.5620089173316956, -2.070064067840576, -0.9474299550056458, -1.5119444131851196, -0.6436983346939087, -0.8693881034851074, 0.15656863152980804, -0.997981607913971, -2.433748245239258, 0.34571871161460876, 0.42919832468032837, 0.7996125817298889, 0.24833287298679352, -1.9774645566940308, -2.0480990409851074, -0.5139411687850952, 0.6807090044021606, 0.6513507962226868, 1.6368653774261475, 0.6012915372848511, 0.67575603723526, -0.8218110799789429, 1.4178271293640137, 0.7320557832717896, 0.16197489202022552, 0.23100359737873077, 3.266725540161133, -0.2776309847831726, -0.03815130889415741, -0.7555537819862366, 0.38515037298202515, 1.2166682481765747, 0.9108473658561707, -0.8554702401161194, 0.006867625284940004, -0.2829347848892212, -1.3507694005966187, -0.43877050280570984, -0.19902591407299042, -0.33255502581596375, 0.1581103503704071, -3.330190896987915, 0.11370556056499481, -1.8799512386322021, 0.8692758679389954, 0.8442325592041016, 0.8062961101531982, -0.41553157567977905, -2.146233558654785, 0.29082244634628296, 1.2171326875686646, -0.19322267174720764, -1.3925284147262573, -2.3016810417175293, -0.06603925675153732, 0.8349682688713074, -0.6805552244186401, 0.008656783029437065, -0.3968297243118286, 1.023537039756775, 0.12833523750305176, -0.2144579291343689, -0.1259184628725052, 0.10486845672130585, 0.359889954328537, -1.051045536994934, -0.3545365035533905, 1.116023063659668, -1.6956177949905396, -0.9524627923965454, -1.1845704317092896, 0.649371862411499, 0.46205025911331177, -0.6169334053993225, -1.0643407106399536, -0.2669104337692261, -1.8750272989273071, -1.72296142578125, -0.5279430150985718, 0.1718774437904358, -0.5510609149932861, 0.16875486075878143, -0.6772305369377136, 1.05674147605896, -0.8981943130493164, -0.9768376350402832, -1.2967267036437988, -0.1925501525402069, -0.9606580138206482, -1.6531070470809937, -1.8756736516952515, 0.7492182850837708, -1.7088922262191772, -1.347853660583496, -0.9703404307365417, -0.7575313448905945, 1.0226510763168335, 1.0237751007080078, -0.7088207006454468, 2.39300537109375, 0.8553304672241211, 0.9260847568511963, -1.5068188905715942, -0.14071276783943176, 1.57176673412323, -0.7423750162124634, -0.45925021171569824, -1.6168946027755737, -0.1483287662267685, -1.0898284912109375, 0.50191330909729, -0.7968090176582336, 1.4103590250015259, -0.23313701152801514, 1.7776471376419067, 1.8304396867752075, -0.36165109276771545, -0.7213801741600037, 0.9778721332550049, -0.04112023860216141, 1.5343211889266968, 1.9740824699401855, -1.257588505744934, -2.0506372451782227, 0.014872223138809204, 2.694139003753662, -0.21868069469928741, 0.25937193632125854, 0.5748604536056519, -1.44278085231781, -0.12463080137968063, 0.5735540390014648, -0.6335859298706055, -1.0557265281677246, -0.9960461258888245, -0.22849039733409882, -0.9233782887458801, -1.1493115425109863, 0.012448976747691631, 0.7980831861495972, 0.7474855184555054, 0.2830270826816559, 1.4345999956130981, -0.38304540514945984, -0.7333437204360962, -1.529433250427246, -0.782141387462616, -0.8709680438041687, -2.562136173248291, -0.5161561369895935, -1.2178864479064941, 0.5046519637107849, -0.3771510422229767, -0.4187859296798706, -0.5606178641319275, -0.6461323499679565, -0.04938782379031181, -1.0326448678970337, -1.1615108251571655, -0.7661359906196594, -0.22149249911308289, 0.4481523334980011, -0.30153611302375793, -1.1753324270248413, -1.3141775131225586, 0.013158593326807022, 1.3082245588302612, -0.19377468526363373, -1.3820245265960693, -0.07526661455631256, 0.08779142796993256, -0.21632297337055206, 0.4173441529273987, 0.6403509974479675, -0.9118756055831909, -0.6065038442611694, -0.08161632716655731, 1.9337600469589233, 0.11151423305273056, -0.45346733927726746, -1.3555680513381958, -0.08373706787824631, -0.13949303328990936, 0.9834596514701843, -0.6992312073707581, -0.4173915982246399, -0.17971864342689514, 0.8494830131530762, 0.33117640018463135, -0.6752930879592896, 0.1332031488418579, 0.7153248190879822, -1.9553139209747314, -0.09120804071426392, -0.1129964143037796, -0.2313721776008606, -1.1743571758270264, 0.30244746804237366, -0.515947699546814, 0.902628481388092, -0.27937158942222595, -0.526821494102478, 0.29227855801582336, 1.6991099119186401, 1.6833739280700684, 0.8789871335029602, 0.4599851369857788, -0.19965162873268127, -1.3099299669265747, -1.1028430461883545, 1.0214369297027588, 1.1443049907684326, 1.0414072275161743, -0.6477003693580627, -0.7664172649383545, 0.17395517230033875, 1.3281129598617554, -0.7055019736289978, 1.0811645984649658, -0.17424961924552917, 1.0985771417617798, -0.07699938118457794, 3.8175768852233887, -0.2840728759765625, 0.8826563358306885, -0.7624343037605286, 1.384454369544983, -0.5272120237350464, 0.08714368939399719, -0.7654063701629639, -0.428549587726593, 0.7428969144821167, -1.2127625942230225, -0.6140851974487305, 0.049500733613967896, -0.730652391910553, -0.7737457752227783, 1.6918927431106567, -1.0671851634979248, 1.031014084815979, -0.7360735535621643, 1.297920823097229, -1.229547142982483, 0.5144288539886475, 1.1139473915100098, -1.1814396381378174, 0.031841352581977844, 1.0495253801345825, 0.7540655732154846, 1.8471163511276245, -0.33223578333854675, 1.2725235223770142, 0.21865829825401306, -0.31565171480178833, 0.031711645424366, 1.4124802350997925, -0.10298377275466919, -0.359651654958725, -0.5340610146522522, -2.0757131576538086, -1.2534918785095215, 2.5124213695526123, 1.3861037492752075, -1.1737182140350342, -0.7076326608657837, 0.6170050501823425, -1.611918568611145, -0.18512433767318726, -0.3740248680114746, 0.7049140930175781, 1.2787339687347412, -1.674928069114685, 0.4446672201156616, -0.4711575508117676, -0.023670319467782974, -1.1773372888565063, 1.0195624828338623, 0.6684948205947876, 1.3672657012939453, -0.0032528340816497803, -0.8810344934463501, 0.9624642729759216, -0.5983752012252808, 0.09108422696590424, -1.1628756523132324, 1.8824081420898438, -1.168993592262268, 0.8876844644546509, -0.6237814426422119, 0.20937007665634155, -0.03806519880890846, 0.08412066847085953, -0.10246391594409943, -0.5341619849205017, 0.11037086695432663, 1.4797550439834595, -1.4716215133666992, -0.5377085208892822, -1.3215339183807373, -0.21338339149951935, -0.16396352648735046, -0.7132366895675659, 0.3824927508831024, -1.3161871433258057, -0.28046008944511414, 0.24782755970954895, 0.30326735973358154, 0.5444788336753845, 0.2214525192975998, 2.062248945236206, 0.25096625089645386, -1.4850162267684937, -1.4345319271087646, 0.41509535908699036, -0.7768678069114685, 2.0482914447784424, -1.4302130937576294, 0.9807248115539551, -0.2521819472312927, 1.7451342344284058, 2.8412892818450928, 0.20759980380535126, 1.8213626146316528, -0.9003399014472961, -0.08812195062637329, -1.2112584114074707, -0.36254292726516724, 0.0072261919267475605, 0.38578376173973083, 0.4516599476337433, 1.384940505027771, -0.6671845316886902, -3.1219096183776855, -0.3332872688770294, 0.18861186504364014, 1.1769381761550903, -0.24435777962207794, -1.005147099494934, 0.5183181762695312, -0.22002024948596954, 1.145323634147644, 0.7823514938354492, -1.8918880224227905, -0.3619537949562073, -1.2698352336883545, 1.0807889699935913, 0.3643880784511566, 1.9390385150909424, -0.2352249175310135, -0.9023593068122864, -0.3567057251930237, 0.9036533832550049, 0.36850589513778687, -0.357216477394104, -0.6117154955863953, 0.2809055745601654, -0.29189327359199524, 0.07451872527599335, 0.9464820027351379, 0.12988194823265076, 0.0176096074283123, -0.49957093596458435, -0.8277942538261414, -1.0296683311462402, 0.2244809865951538, 0.616908848285675, 0.047421377152204514, -0.290835976600647, -1.8407419919967651, 1.0860928297042847, -0.21757721900939941, -0.9350022077560425, -0.6265915632247925, -1.4171727895736694, 0.4881907105445862, -1.0678855180740356, 2.049438238143921, -0.9822302460670471, 0.3336021602153778, -0.46552589535713196, 0.49463704228401184, 0.026887385174632072, -0.0528981015086174, 0.3222961127758026, -0.09897206723690033, 1.9001981019973755, 1.2513896226882935, 0.2743876576423645, -0.2337752729654312, 0.9078120589256287, 1.019051432609558, 1.4971065521240234, -0.48326313495635986, 1.089516520500183, 0.0792846605181694, -3.0497453212738037, -1.6437255144119263, -0.2739812135696411, -1.8334314823150635, 1.033726692199707, 1.084720253944397, -0.7007582783699036, -1.0358071327209473, 0.5437866449356079, 0.5204371213912964, 0.6145456433296204, 0.5236021280288696, 0.5333594679832458, 1.2187825441360474, 1.0673350095748901, 0.6058403849601746, 0.8113960027694702, 1.1303958892822266, -0.9281395673751831, -0.6144806742668152, -0.05308102071285248, -0.5624423623085022, -0.07456927001476288, -1.1371842622756958, -2.583763360977173, 0.4258396625518799, 0.9032552242279053, -0.33417391777038574, -0.42199212312698364, -0.5253379940986633, -1.5049545764923096, -0.5350515842437744, -0.2264014333486557, 1.7194478511810303, 1.3695725202560425, -0.4337662160396576, -0.6185752749443054, 1.0244508981704712, -1.587594747543335, -0.6965362429618835, -0.7333723306655884, -0.6236834526062012, 0.7441433668136597, 0.8023792505264282, 1.0107166767120361, 0.29164499044418335, 1.1421782970428467, -0.6841606497764587, -0.0011163951130583882, -0.7088409662246704, -0.7804185152053833, -1.0870682001113892, -0.646136462688446, 1.0230475664138794, 0.9724679589271545, 0.27546364068984985, 0.17798976600170135, -0.7500864863395691, 0.868706464767456, -0.4685506224632263, 0.789320707321167, -0.4535468816757202, -0.5080663561820984, 0.9715391397476196, 0.10029570758342743, -0.25464925169944763, -0.6214497685432434, -1.9257975816726685, 0.017101014032959938, -1.1728713512420654, 1.6401673555374146, -0.6000824570655823, 0.41357821226119995, 0.936465859413147, -0.23479081690311432, 0.7019122838973999, -0.6010777950286865, 0.0743817612528801, 1.5073102712631226, 0.35572388768196106, 0.05019710958003998, 0.8786671161651611, 0.3030974268913269, 0.28078463673591614, 0.14526881277561188, -1.698249101638794, 0.8329716324806213, -0.9793254137039185, 0.28260567784309387, 0.9991186261177063, 0.5617729425430298, -0.5775216817855835, -0.38036367297172546, 0.08892876654863358, -0.41781890392303467, -0.512925922870636, -1.0010484457015991, 1.2449162006378174, 1.4215788841247559, -0.7921310663223267, 0.8327047228813171, -0.8032026886940002, -0.0279726330190897, -0.15364596247673035, -0.8008348941802979, 1.0206838846206665, 1.0432218313217163, -0.11042647808790207, -1.490560531616211, -2.64201021194458, 0.6266217827796936, 0.21821634471416473, -0.9043789505958557, 1.5370022058486938, 0.3541962206363678, -0.9669926166534424, 0.17024676501750946, -0.7328786849975586, 0.970491886138916, 0.09839431941509247, -0.7465084791183472, -0.6276586651802063, -0.3229442238807678, -1.2847000360488892, -0.1785767674446106, -2.276545524597168, 2.5013365745544434, -0.4942740201950073, 0.06668584048748016, 0.4329202175140381, -0.664845883846283, -1.6846868991851807, -1.088637351989746, -0.19001425802707672, 0.7940483093261719, -0.3376000225543976, -0.1715579628944397, -1.2486342191696167, 0.22060918807983398, 0.30843839049339294, -0.3329911530017853, -0.6473398804664612, 0.9275503754615784, 0.48837506771087646, -0.10532962530851364, 0.08582931756973267, -0.44657281041145325, -0.7409261465072632, 0.7797205448150635, -1.3339087963104248, 1.1152085065841675, -0.34302133321762085, 0.062489550560712814, -0.09248258173465729, 0.5100858807563782, 1.3540014028549194, 0.3138512074947357, 0.47385868430137634, -0.7016229033470154, 1.2202235460281372, 0.34876111149787903, -0.8697247505187988, 1.0165210962295532, -0.8123139142990112, 0.7241894602775574, 0.5926613211631775, -0.7488417625427246, 1.7084107398986816, -0.3935309946537018, -1.4667247533798218, 1.346412181854248, 2.235616683959961, 0.8739660978317261, -0.07811961323022842, 0.6018924117088318, 0.8002690672874451, -1.1483269929885864, -0.9114819765090942, 0.41988644003868103, -0.5731818675994873, 1.5405393838882446, 0.5785313248634338, -1.7494045495986938, -0.2588336765766144, 0.10025894641876221, 0.9013952612876892, -0.9117998480796814, 0.013803007081151009, 0.8900853991508484, 0.052903544157743454, -0.31796273589134216, -0.48511916399002075, -0.2955450117588043, 0.6183197498321533, -1.2890180349349976, -1.171860694885254, 0.7661337852478027, -0.8794174194335938, 0.18332581222057343, 1.3743553161621094, 2.0552151203155518, 0.749770998954773, 0.9510264992713928, 0.9767481088638306, 0.035652995109558105, -0.6547848582267761, 0.425011545419693, 0.09501637518405914, 0.5306811332702637, -0.04549748823046684, 0.8677683472633362, 0.5774314403533936, -0.4889366328716278, -0.9892330765724182, -0.7643811106681824, -0.3492182195186615, -0.14139264822006226, -0.25196582078933716, -0.687187910079956, -1.3470215797424316, 0.35245057940483093, 0.4397200345993042, -2.8895604610443115, -0.7711628079414368, -0.29906973242759705, 0.8506162166595459, -0.5878224968910217, -0.5869587063789368, 1.2632235288619995, -1.1882389783859253, 1.1177682876586914, 0.34842050075531006, 0.06501196324825287, 1.2868423461914062, 1.0872136354446411, 0.00788786355406046, -1.727501630783081, 0.67678302526474, -1.0732197761535645, 0.5022897124290466, 1.4802438020706177, 0.9963944554328918, 0.4221542775630951, 0.4608287215232849, 0.6107958555221558, -1.7678576707839966, 0.48868662118911743, -0.3783576190471649, -0.252003014087677, -0.2966262996196747, 0.11036111414432526, -1.478644847869873, -0.698276698589325, -0.043181952089071274, 1.4033029079437256, -0.14469611644744873, 1.3826979398727417, -0.7531737685203552, 1.6152724027633667, 0.6605804562568665, -0.7008531093597412, 1.1479604244232178, 0.046131931245326996, -0.9208345413208008, 0.0726194754242897, -0.38239777088165283, -1.73727285861969, -0.8688423037528992, 1.5321398973464966, 0.6973554491996765, 0.10487648844718933, 1.6647635698318481, 1.4548265933990479, 0.5106924772262573, -0.029248328879475594, 2.2017035484313965, -0.1805552989244461, -0.2003144770860672, 1.167534351348877, -1.923728108406067, -0.8764873743057251, 0.40192708373069763, -0.4187663197517395, 0.9214844703674316, -0.08797652274370193, -0.1267508715391159, 0.5970698595046997, 0.22776232659816742, -1.7368000745773315, 0.8116309642791748, -1.5426563024520874, -0.5106444954872131, 0.9502874612808228, -0.19370438158512115, -2.0478436946868896, 0.15630164742469788, -0.014777651987969875, -1.487627387046814, 1.9173755645751953, 0.18654674291610718, 1.3655648231506348, -0.14189499616622925, 1.5043649673461914, 0.03037426620721817, 0.7139806747436523, -0.28987252712249756, 0.5084320902824402, 0.09686215221881866, 2.846754550933838, -0.5934584140777588, 0.912565290927887, 0.270378440618515, -0.7397794723510742, 0.8225747346878052, 0.45810213685035706, -1.0258941650390625, -0.7712847590446472, -0.2755293548107147, -0.8201603889465332, -0.7563838958740234, -1.0746488571166992, 0.9533759355545044, 0.29683300852775574, 0.35616952180862427, -0.28669536113739014, -1.6256401538848877, 2.1680612564086914, 2.7023227214813232, 2.732448101043701, 0.02826308272778988, -2.155369281768799, 1.1347053050994873, -0.7455452680587769, -0.19570451974868774, -1.2940846681594849, -0.3439343571662903, -0.025832664221525192, -1.7687417268753052, -0.5128875970840454, -0.25759831070899963, -0.010122112929821014, 0.7117127776145935, -0.07826947420835495, 0.27464962005615234, -1.6639418601989746, -0.2953692376613617, 0.6259552836418152, 0.2598327696323395, 0.6516152024269104, 0.17787782847881317, -0.2567562162876129, 1.2882471084594727, -0.09350863099098206, 0.20915378630161285, -0.002849765121936798, -0.6457840800285339, -0.7342748641967773, 1.6521522998809814, 1.2332046031951904, 0.417836457490921, -0.7616395354270935, 0.19196023046970367, -0.7001658082008362, -0.2408798784017563, -0.4465820789337158, 1.0120340585708618, -0.2154955416917801, 0.7737051248550415, -1.6157547235488892, 2.1170454025268555, 1.1416447162628174, 0.026545677334070206, 0.053686488419771194, 1.4098670482635498, 0.6446442008018494, 0.563075065612793, -1.1373093128204346, -1.0100359916687012, 0.5332883596420288, -1.094605565071106, -0.05240442976355553, 0.27984514832496643, 0.30829918384552, 2.010218620300293, 0.17534542083740234, -1.1528890132904053, -1.4401267766952515, 1.2118669748306274, -0.39925941824913025, -0.6069378852844238, 0.7188378572463989, -0.3738347887992859, 0.6434589624404907, -0.4572182893753052, 0.6313313841819763, 0.07463761419057846, -0.1819087415933609, 0.39073827862739563, 1.0392347574234009, -0.3453008830547333, 1.3214898109436035, -0.9116746783256531, 0.47141194343566895, 1.187183141708374, -0.06381353735923767, 0.12474001944065094, -0.2227240651845932, -0.43419840931892395, -1.1770583391189575, 0.8793802261352539, -1.0235258340835571, 0.1069503203034401, 0.6836442947387695, -0.27807003259658813, -1.0526864528656006, -0.5693355202674866, -0.27242767810821533, 0.9482782483100891, -0.22023630142211914, -0.27930697798728943, 0.5327409505844116, 0.43462052941322327, 1.2021721601486206, 0.8879806399345398, -0.5283601880073547, -0.5885714888572693, 1.6304900646209717, -1.3338274955749512, 0.03929121419787407, 0.5224633812904358], [-0.5076722502708435, 0.3295074999332428, -0.5765596628189087, 0.3987913131713867, 0.7551706433296204, 1.5031039714813232, -2.4777183532714844, 0.3115084171295166, 0.6650351881980896, -0.9671650528907776, -0.3306528627872467, -0.6528818607330322, -1.9930691719055176, -1.2211376428604126, -0.16724902391433716, 0.10753947496414185, -2.333944082260132, -0.5380202531814575, 0.21715012192726135, 0.4177424907684326, -0.6450997591018677, -0.13879913091659546, 0.2986464500427246, 0.49139028787612915, -0.8638589978218079, 1.2413215637207031, -0.3481036126613617, -1.4458009004592896, 0.7035156488418579, 1.2512813806533813, 1.6751108169555664, -1.5516754388809204, 1.6951894760131836, -0.8413688540458679, 0.8799856901168823, -2.452333927154541, 0.8443700075149536, -1.5466384887695312, -0.0767858549952507, -0.5425275564193726, 0.613084614276886, -1.0308717489242554, -0.6467806696891785, 1.8945575952529907, -0.6240607500076294, -1.5360815525054932, 1.0992159843444824, 0.41106805205345154, 0.43199774622917175, -0.36648890376091003, -1.5558607578277588, -0.7803587317466736, -0.6413140296936035, -0.868897020816803, -0.05393815413117409, 2.2561535835266113, -1.265444040298462, -0.5755056142807007, -0.34359028935432434, -0.5860437154769897, -0.4912945032119751, -0.7888992428779602, 0.6535316705703735, -0.06750733405351639, -0.9414452910423279, -1.0123648643493652, -0.04092545062303543, -0.6245553493499756, 0.34811294078826904, -0.5919038653373718, 0.5503974556922913, 1.5926355123519897, 0.6160560846328735, 0.05945684015750885, -0.06146784871816635, 0.03697532042860985, -0.6450305581092834, -0.35739219188690186, -0.3883909285068512, -1.8001856803894043, -0.11467296630144119, 0.36760514974594116, -0.19487936794757843, 0.7909744381904602, -0.5799209475517273, 0.9189062118530273, -1.2617682218551636, -0.10775606334209442, 0.1355060487985611, -2.1393840312957764, -0.682100772857666, 0.7406759858131409, 1.4535918235778809, -2.2689454555511475, 0.6631183624267578, 0.5891786813735962, 1.0646119117736816, -0.0058636353351175785, 1.6071499586105347, 0.8687455058097839, -0.3353065550327301, 0.5125445127487183, -0.0029801889322698116, 0.023111997172236443, 0.4745485782623291, -0.8737747073173523, -0.3053113520145416, -1.5064162015914917, -1.688315749168396, 1.9201631546020508, -1.0686858892440796, 0.8285573720932007, 0.11038361489772797, -0.07059378921985626, 0.27236443758010864, -0.22596479952335358, -0.23643450438976288, 1.6136889457702637, -0.3827114701271057, 1.472389817237854, -0.8689796924591064, -0.7003029584884644, 0.6432101130485535, -3.3215224742889404, -0.5894841551780701, 1.1807746887207031, 0.4367733597755432, -0.8907970190048218, -1.1978895664215088, -0.7981871366500854, 0.08490428328514099, 0.10781754553318024, -0.8492147326469421, 0.43566396832466125, 0.35613468289375305, 1.5356463193893433, -1.8604459762573242, -1.4532010555267334, -0.37785419821739197, -1.3630155324935913, -1.3410943746566772, -0.056582123041152954, 2.308957815170288, 0.5838643312454224, 0.5923246145248413, 0.1781669706106186, -0.7104393839836121, 0.3577604293823242, -1.6441705226898193, -0.23409776389598846, 0.6395185589790344, -0.2555033266544342, 0.7040833234786987, 0.5295401215553284, 0.038444388657808304, -0.2745119035243988, -0.29330772161483765, 0.4068422019481659, -0.6512036919593811, -0.1260816752910614, -0.11870749294757843, 1.0820989608764648, -0.5143720507621765, 0.7259861826896667, -0.03947210684418678, 0.9464583992958069, 0.021547408774495125, -0.014288938604295254, 1.842018961906433, -1.167190670967102, 0.051400817930698395, 0.9806851148605347, 1.5506560802459717, -0.24695636332035065, -1.583018183708191, -0.19075697660446167, 0.7653470635414124, 0.6705055832862854, -0.925387978553772, -1.7614240646362305, -0.6508592963218689, 0.051526956260204315, 0.6479239463806152, -0.4416145980358124, -0.8739017248153687, -0.954574704170227, 0.09099823981523514, -0.059010908007621765, 0.20844577252864838, 1.2570699453353882, 1.3460021018981934, -2.185051202774048, -1.8468056917190552, 0.5054970979690552, 0.8849540948867798, -1.0073210000991821, -0.13401643931865692, -0.7257261276245117, -0.8957520723342896, -2.1807069778442383, -0.378495991230011, 0.05422215908765793, -0.7781155109405518, 0.8721078634262085, -0.7396274209022522, -0.3545241057872772, -0.3931795060634613, 0.3499191403388977, 0.30636441707611084, 1.147919774055481, 1.1880890130996704, -0.5573583841323853, 0.4286183714866638, 0.9188602566719055, 0.41179320216178894, -0.27056577801704407, -0.6226650476455688, 0.6351020336151123, 0.8037763833999634, -0.2398661971092224, -1.5363482236862183, 0.8117533326148987, 0.0397428423166275, -3.0890562534332275, 0.28584572672843933, 0.32196709513664246, -0.6515552997589111, 0.37940678000450134, 0.5570270419120789, -0.10848893970251083, -1.0374070405960083, 0.44110679626464844, 0.6315324306488037, 0.7044638991355896, -0.2731185257434845, 0.1257125586271286, 0.2620142996311188, -1.5056692361831665, -0.7417128682136536, -0.3240024149417877, 1.3087918758392334, -0.32871466875076294, -0.29367005825042725, 1.6229853630065918, -0.433316171169281, -1.6912788152694702, -1.5501816272735596, -0.9597837328910828, -0.7764827013015747, 2.108064889907837, 0.7159088253974915, -0.001757955295033753, -0.14424724876880646, 0.8560507297515869, -0.5863069891929626, 1.1558575630187988, -2.1052305698394775, -0.1979694366455078, 0.18118587136268616, 0.41629546880722046, -0.5393280982971191, 0.04083575680851936, -1.954331398010254, -0.7303395867347717, 2.0568289756774902, -0.6957846879959106, -0.4263988137245178, -1.4947532415390015, -1.5597916841506958, 0.4508391320705414, 0.19260570406913757, 0.6260285973548889, -0.28257477283477783, 0.9297534227371216, 0.1055617704987526, -2.0125303268432617, 0.08855217695236206, -0.11194195598363876, -0.22910217940807343, -0.7221494913101196, 1.2472193241119385, 0.3005521297454834, -0.5414332747459412, 0.8742783069610596, 0.5721688866615295, 0.32541415095329285, 0.048136401921510696, -0.8795724511146545, -1.4594963788986206, -9.437782864551991e-05, 0.5654012560844421, -2.223484992980957, -0.7674563527107239, 0.8796064257621765, 0.8872011303901672, 1.1735705137252808, -0.20280028879642487, 0.09512587636709213, 0.2744773328304291, -1.017350673675537, 0.0345364548265934, 0.2620425522327423, -0.35510382056236267, 1.6594220399856567, -1.553275227546692, 0.13700610399246216, -1.736916184425354, -0.8250470161437988, -0.743951141834259, -2.236921548843384, 0.2931058704853058, -1.807381272315979, -0.4623923897743225, -0.6825817823410034, 1.1787984371185303, 0.9980560541152954, -1.379523754119873, 0.652793824672699, -0.029203685000538826, 1.259761095046997, 1.6338105201721191, 1.1859123706817627, 0.3970737159252167, -0.5786923170089722, 0.28904473781585693, -1.1485124826431274, -0.08670532703399658, 0.9525333046913147, -1.7436033487319946, 1.431840181350708, 1.0299104452133179, -0.42936971783638, -0.3371986150741577, 0.19019965827465057, -1.7514292001724243, 0.04508885741233826, -0.9992160797119141, -2.020831823348999, 1.3938544988632202, -1.2935291528701782, -0.4918927550315857, -0.6265085935592651, -0.16991624236106873, -1.2542015314102173, -2.037628650665283, -0.3545707166194916, -2.719984292984009, -0.800209105014801, -1.7947362661361694, 0.7599194049835205, -0.9724491834640503, 0.5223994255065918, -0.5353276133537292, -0.7534372210502625, 0.9237145781517029, -0.4468846023082733, -0.6896195411682129, 0.14470668137073517, 0.12229347229003906, 0.5107643604278564, 0.4115375280380249, 0.2272322028875351, 0.8626925349235535, -0.6073311567306519, -0.22543218731880188, 0.9655116200447083, -0.17231811583042145, -0.5235270857810974, -1.3467093706130981, 0.867344856262207, -0.4455757737159729, -0.016698313876986504, 0.2369101196527481, -0.7460811734199524, -0.7984164357185364, -1.6129168272018433, -0.35290151834487915, 0.20437228679656982, 1.8707135915756226, 0.33062925934791565, -1.5781148672103882, -0.027888387441635132, 0.30110040307044983, -0.16538389027118683, 0.22589211165905, 1.8838330507278442, -0.6463972330093384, -0.6201420426368713, -0.12391453236341476, -0.45253634452819824, -0.9580293297767639, -0.023653024807572365, -1.6024305820465088, 0.9851304292678833, 0.9021686911582947, 0.026235202327370644, -0.4855283200740814, 1.3368183374404907, -0.12285841256380081, -0.7888354659080505, 0.33910781145095825, 0.465017706155777, 0.5371925234794617, 1.0592318773269653, -0.5712818503379822, 0.8646572232246399, -1.1495405435562134, -0.8591094017028809, 0.6620974540710449, 0.518314003944397, 1.0363709926605225, -2.357870578765869, -0.6809250712394714, 2.203433036804199, 1.139561653137207, 0.9371998310089111, 0.31256237626075745, -0.9411088824272156, -1.6043795347213745, -0.6352351307868958, 0.4632064700126648, 0.5452331900596619, -1.6904406547546387, -0.22005702555179596, -0.2946975827217102, -0.4861098527908325, -1.074170470237732, -0.4181089401245117, 0.16227713227272034, 0.6109703779220581, 0.18854497373104095, 1.1728094816207886, -0.022280437871813774, -0.5405448079109192, -1.874306082725525, -0.26012706756591797, -0.11359056830406189, -0.15720444917678833, 1.1377028226852417, 1.081135630607605, -0.5643784999847412, 0.6765827536582947, -0.16221213340759277, 0.20081105828285217, 1.3325419425964355, -1.075095534324646, -2.3039705753326416, -0.8250939249992371, 1.9375669956207275, -1.2105909585952759, -0.20023052394390106, 0.6130416989326477, 0.3491066098213196, 0.390621155500412, 0.40931436419487, 0.8744664192199707, 1.3903957605361938, -1.054092526435852, -0.7551996111869812, -0.6790536642074585, 2.3382351398468018, -0.568902850151062, 0.580414891242981, -1.4301061630249023, 0.5138980150222778, 0.8424822092056274, -0.45929843187332153, 0.4451957643032074, 0.07938133925199509, 0.21410255134105682, -1.5097049474716187, 0.9247586727142334, 0.6289104223251343, -0.13591550290584564, -0.3236609399318695, -0.3061167299747467, -0.8783915042877197, 0.6824331283569336, 1.605473518371582, 0.12939684092998505, -0.006019110791385174, -0.07999899238348007, 0.5309519171714783, 0.8849771022796631, -0.8217654824256897, 1.2770514488220215, 0.18934930860996246, 0.44217368960380554, -0.812158465385437, -0.7171453237533569, 0.015725694596767426, -0.16137824952602386, -0.6513194441795349, -0.30580252408981323, -0.5183898210525513, -0.14882805943489075, 0.07390327006578445, 0.557323157787323, 1.4090181589126587, -1.234062910079956, 0.1255134642124176, -1.1178981065750122, 0.47170189023017883, 0.6684134602546692, 0.7541823983192444, 1.0223511457443237, -1.8941227197647095, 0.929542601108551, -0.893526017665863, 0.492460161447525, 0.8030509352684021, 0.6626385450363159, 0.4766848385334015, -1.133357048034668, 1.1876118183135986, 0.49057096242904663, 2.175304651260376, -0.2395462989807129, 1.1097378730773926, -0.3854336738586426, -1.6459543704986572, 1.8035305738449097, -0.39237555861473083, 0.6615279316902161, 1.158308744430542, 1.7417511940002441, 0.5203531980514526, 0.16731029748916626, 0.7157321572303772, 0.5400214195251465, -0.6532490849494934, 1.6852351427078247, -0.9303658604621887, 1.8951352834701538, -1.0050827264785767, -0.30409300327301025, 2.417919635772705, -0.7647150158882141, 1.0543012619018555, 1.213197112083435, 0.6350952982902527, -0.6282010674476624, 2.265836000442505, 0.0049712080508470535, -0.18997453153133392, -1.3264051675796509, -0.23762956261634827, -1.6420202255249023, 0.7323706150054932, -1.6638705730438232, -0.4141419231891632, -1.2626841068267822, 0.023613212630152702, 1.7568607330322266, 1.6160979270935059, -1.1710455417633057, 0.87237948179245, 1.5539571046829224, 0.8491361737251282, -0.31537142395973206, 1.7121678590774536, -0.3701190948486328, -0.4283197820186615, -1.9781228303909302, 0.384391725063324, -1.18922758102417, -0.4733484387397766, 0.24524010717868805, -0.3499743938446045, 2.2462871074676514, 0.41884034872055054, 0.8518745303153992, 0.27799633145332336, -0.09189685434103012, 0.6211026906967163, 0.6989103555679321, 2.5500118732452393, -1.302901268005371, 0.053208671510219574, 0.38823267817497253, -0.42690524458885193, -1.2904012203216553, -1.061631202697754, -0.4744657576084137, -1.155110478401184, -0.7438642978668213, 0.8018167018890381, 0.8549890518188477, -0.38742053508758545, 0.3263353407382965, -0.07710345089435577, 1.218812108039856, -0.18378137052059174, 0.33559733629226685, -0.4431859254837036, -0.11376915127038956, -1.2148728370666504, 2.0622031688690186, -0.10473955422639847, 0.4650966227054596, 1.9632035493850708, -0.5148488283157349, 1.1533011198043823, 0.7043080925941467, -0.9329457879066467, 1.5483592748641968, 0.11368682235479355, 1.1644618511199951, 0.07460737973451614, 1.484344244003296, -0.5580748915672302, 1.2515584230422974, 0.23814788460731506, -1.5415306091308594, 0.31795307993888855, 0.7779600620269775, -0.13333174586296082, 0.3780879080295563, 1.448825478553772, -0.02745749056339264, 1.1170146465301514, 0.07662711292505264, -0.2563382387161255, -0.45711347460746765, 1.3893448114395142, -0.5110277533531189, -1.7589222192764282, -1.4003316164016724, 0.6151688694953918, -0.3436819016933441, 0.9264429211616516, 0.022417858242988586, 0.40580445528030396, 1.785525918006897, 1.4493898153305054, 0.8898245096206665, 0.8389213681221008, -0.910805344581604, 0.986352264881134, 0.25179946422576904, -0.3689585328102112, 0.7572027444839478, -0.9571465849876404, 0.33489322662353516, -1.2389055490493774, -0.11513085663318634, 0.5849551558494568, 0.3813018202781677, 0.09477398544549942, 0.6094329953193665, 0.5177618861198425, -0.10196815431118011, -0.5974968671798706, 0.4542296528816223, 0.16473059356212616, -0.8394041657447815, 0.049360405653715134, 0.06591202318668365, 0.2140255868434906, 1.3378422260284424, -0.8679993748664856, 0.509427011013031, -0.5577031970024109, -0.17801548540592194, 0.37011319398880005, -1.2836993932724, -1.1227487325668335, -1.0730856657028198, 0.3344936668872833, 1.9387353658676147, 1.1519362926483154, -1.8209905624389648, 1.857229232788086, -0.6754608750343323, 2.25018048286438, -0.9744462370872498, -0.2038763165473938, 0.3492749333381653, 1.6586222648620605, -0.7926307320594788, -1.2340316772460938, 0.35591211915016174, -0.16770683228969574, -1.0121002197265625, -1.2945469617843628, -0.706906795501709, 0.8803707361221313, 0.44071558117866516, 2.1114256381988525, 1.524328351020813, -0.3144918382167816, 1.822874665260315, 0.37839898467063904, 0.01374382246285677, -0.43900126218795776, -0.6588144302368164, 0.30044859647750854, -1.0058475732803345, 0.17413194477558136, 1.5576562881469727, -0.9946197271347046, -0.20097023248672485, -0.5606322288513184, -0.28011780977249146, 0.4804214835166931, 0.513209879398346, 0.5024476647377014, -0.3372563421726227, -0.028669528663158417, -0.650016188621521, -0.18481869995594025, 0.21930031478405, -0.5826308131217957, -1.2896713018417358, -0.6297056674957275, 0.7302324175834656, 1.1243162155151367, -1.8015313148498535, 1.3712992668151855, -0.07602579891681671, -0.8118701577186584, 0.48778510093688965, -0.6062817573547363, -0.5308619737625122, -0.3766879439353943, 0.9475910067558289, -0.5527966022491455, -1.1796122789382935, 0.6527622938156128, -0.6793569922447205, 0.8625048398971558, -0.5596958994865417, -0.18471863865852356, 0.476019948720932, 1.780126690864563, -0.4787845313549042, 0.3133336007595062, -0.13502754271030426, -1.6625250577926636, 1.3113908767700195, -1.0484422445297241, -0.5805763602256775, 1.543470025062561, -2.0694026947021484, 0.9721590876579285, 0.10314217954874039, -0.4085790812969208, -0.08967608958482742, 0.3907639980316162, 1.109127163887024, 0.1787140667438507, -2.4277100563049316, 2.1836495399475098, 0.543923556804657, -0.6857885122299194, -1.2218738794326782, 1.3599464893341064, 0.06395189464092255, 0.12552854418754578, 1.1850067377090454, -0.7689457535743713, 1.1149410009384155, -0.20237784087657928, -0.20285944640636444, 0.32375282049179077, 1.1436598300933838, 0.9130571484565735, 0.14705893397331238, -0.8354291319847107, -1.401066780090332, -0.07709363847970963, 0.3791936933994293, -0.23008932173252106, 0.006453576032072306, -0.3066304326057434, -0.5427988171577454, 0.494995653629303, -0.29202884435653687, -0.6221784949302673, -1.757310152053833, -0.044533345848321915, 0.1886054426431656, 0.7141910791397095, -0.23401305079460144, -0.3196200132369995, -1.6892319917678833, 1.1068856716156006, 1.3428577184677124, -0.8003717660903931, 0.38075119256973267, 1.2661080360412598, -0.5822027921676636, 0.7475541830062866, 1.5534465312957764, -0.4558851420879364, 0.03645608201622963, 0.5541250705718994, 0.260993629693985, -0.6661850214004517, -0.717750608921051, -0.2897011935710907, -0.3756664991378784, -1.3167710304260254, 0.6412648558616638, -0.37555360794067383, -1.2190929651260376, 0.2740515172481537, -0.9172971844673157, -1.3255802392959595, -1.6942224502563477, -2.5357704162597656, -1.0987310409545898, 0.51646488904953, 1.1510463953018188, -0.820778489112854, 1.1643942594528198, -0.482339084148407, 0.38296404480934143, -0.02956843376159668, -0.7214258909225464, -0.003930095583200455, -0.9228796362876892, -0.4478778541088104, 0.09049712121486664, -3.6431474685668945, 0.0703214704990387, 0.4846782982349396, -1.4072959423065186, 0.6938667297363281, -0.6506412625312805, 0.5716537833213806, 0.17313152551651, 0.8302977681159973, 0.529105007648468, 0.701310396194458, 1.240123987197876, 0.48120564222335815, -2.0855367183685303, 1.1055065393447876, -0.5847187042236328, 1.4522687196731567, -0.7784900069236755, -0.009486593306064606, -0.13797712326049805, 0.2513955235481262, -0.5680199861526489, 0.41010862588882446, -1.1600335836410522, -0.589837372303009, 0.02057289518415928, 2.224001169204712, 1.592413306236267, -1.245405912399292, 0.3315194845199585, 1.3547698259353638, -0.3909527659416199, 0.8199567794799805, -0.39150914549827576, -0.5861523151397705, -0.23393872380256653, -0.01880853995680809, 0.4763661026954651, -1.1188551187515259, -0.4359133243560791, 1.9770430326461792, -1.596297025680542, 0.232195645570755, 1.3864847421646118, 1.2543094158172607, 0.0262353103607893, -1.2608134746551514, 0.8977541923522949, 0.19677545130252838, 0.8106864094734192, -0.6008357405662537, -0.42163655161857605, -1.8522013425827026, -0.828698992729187, -1.087084412574768, -0.6217713356018066, 0.8745908737182617, -0.8019208312034607, -0.2680414617061615, 1.3654931783676147, 0.3953549861907959, -0.3238984942436218, 0.21971328556537628, 0.8320370316505432, 1.0420501232147217, 1.3336176872253418, -0.2788970470428467, -1.5727908611297607, -1.0454540252685547, 0.026296546682715416, -1.6654866933822632, 0.6592429280281067, -2.052638292312622, -0.04038817435503006, -0.5652497410774231, -1.4379289150238037, -0.977243959903717, -0.0628647431731224, 0.05061188340187073, 0.29866257309913635, -0.9448436498641968, -0.3090441823005676, -0.189385786652565, 0.970952033996582, 0.24534322321414948, -0.1364261507987976, 0.4656844735145569, -0.20998184382915497, 0.18842892348766327, 0.3000689148902893, -0.5179846286773682, 0.6341716051101685, 0.28864312171936035, -2.1883957386016846, -0.5488659739494324, -1.4369573593139648, 0.020334068685770035, -1.7501579523086548, 0.3623614013195038, -0.15924784541130066, 0.5707769393920898, 1.118959903717041, 0.8849028944969177, -1.2298941612243652, 0.41090238094329834, -0.333667129278183, -0.6458812952041626, 0.9397874474525452, -0.6889258027076721, 0.31762105226516724, -0.9616355895996094, 0.5850569605827332, -2.2689714431762695, 0.20852062106132507, -0.4716271460056305, -0.5884634256362915, -1.3463928699493408, 0.6626561284065247, -0.33737438917160034, 0.6091002225875854, -1.199725866317749, 1.1464437246322632, -0.8833447694778442, -0.9480326175689697, -0.8318987488746643, -0.4966872036457062, 0.2678200304508209, 0.24422693252563477, -0.41737231612205505, -0.6152722239494324, -2.0751419067382812, 1.1108813285827637, 2.4316649436950684, 1.2055344581604004, -1.7235896587371826, 0.6991803646087646, 0.02862517721951008, -2.2057228088378906, -2.336575984954834, -0.6487895846366882, 0.9464751482009888, 0.4725252091884613, -0.592350423336029, 0.25579822063446045, -1.1664186716079712, 1.0161150693893433, -1.2060089111328125, -1.0025660991668701, -0.3512636423110962, 1.2789585590362549, 1.0016695261001587, 2.066681146621704, 0.607393205165863, -1.8286298513412476, 0.05609167739748955, 0.7059519290924072, 0.23879435658454895, -1.5052074193954468, 1.2685482501983643, 0.6652894020080566, -0.44827836751937866, -1.0062389373779297, 0.5808660984039307, 0.3163994550704956, -0.2197256237268448, -0.967422366142273, 1.2681808471679688, 0.8101178407669067, -0.36361661553382874, -1.0364511013031006, -0.8124443888664246, 0.20000438392162323, -0.20949062705039978, -1.6226056814193726, 1.3212240934371948, -0.36377260088920593, -1.2090424299240112, -0.5886685848236084, -1.9812496900558472, -0.45833343267440796, 0.9563440084457397, -2.2878262996673584, 0.00013104145182296634, 0.19680359959602356, -0.07565855979919434, 0.8872888684272766, -0.5323119163513184, 0.2969117760658264, 1.3302414417266846, 0.441101998090744, 0.494994193315506, 0.801982581615448, 0.7381986975669861, -1.0253747701644897, 0.3809431493282318, -0.0006818958790972829, 0.22621777653694153, -2.0452070236206055, 0.518584132194519, -0.353487104177475], [1.7030192613601685, 0.3212967813014984, -1.1792562007904053, 0.339800626039505, -1.2594913244247437, 1.067183494567871, -0.09938066452741623, -0.8080609440803528, -1.0452654361724854, -0.3148491680622101, 0.18809828162193298, -0.16306284070014954, 0.5252671837806702, 1.6978000402450562, 0.06099870428442955, 0.4771696925163269, -0.30262917280197144, 0.031086377799510956, -0.07800234109163284, 0.21781276166439056, 1.0956073999404907, 0.4005708694458008, 0.8488012552261353, -1.2002582550048828, -2.069645643234253, -1.9104276895523071, 0.3578343093395233, -0.137405127286911, 0.5645177960395813, -0.6391862034797668, 0.25130224227905273, 0.6964136362075806, -1.4137701988220215, -0.983277440071106, -1.1509867906570435, -2.062875986099243, -1.3056540489196777, 1.2654155492782593, -0.07341417670249939, -0.36457160115242004, 1.3805394172668457, -1.3415124416351318, 0.0321388877928257, 0.15041851997375488, 0.3525823950767517, -0.45935073494911194, 1.0428318977355957, -0.8357469439506531, -0.34743642807006836, 0.07616689801216125, 0.5345504879951477, 0.08254259824752808, -2.042680263519287, -0.8305200338363647, 0.2613139748573303, 0.7159039974212646, -0.6730892062187195, -0.8732625842094421, 0.29473876953125, -1.2066733837127686, 1.1609835624694824, -0.932174563407898, -1.2457534074783325, -1.5643402338027954, 1.2971165180206299, -0.9826835989952087, -0.40974804759025574, -0.6528839468955994, 0.2981323003768921, 1.9130712747573853, 0.6686937212944031, 0.5704194903373718, -0.6656948328018188, 0.6717234253883362, -0.25590863823890686, -0.10247467458248138, 0.4096883237361908, -1.5891499519348145, 1.3269798755645752, 0.3548310697078705, -0.6217786073684692, -2.0757339000701904, -0.547109067440033, 1.3945157527923584, -1.0226079225540161, -0.12046641111373901, 0.9018742442131042, 1.0252858400344849, 0.5123217105865479, -0.015765629708766937, -0.4719478189945221, -0.7200226187705994, -2.7799580097198486, -0.6637143492698669, 0.6916624307632446, -1.8675204515457153, 1.4653851985931396, -0.29323968291282654, 1.6436301469802856, 0.06720256060361862, -0.6398550271987915, -0.07758637517690659, -0.008602752350270748, -0.7842774391174316, -1.1501410007476807, -0.23257778584957123, 1.3596696853637695, 0.348698228597641, 2.1440625190734863, 0.8888312578201294, -1.6812114715576172, -0.862751841545105, 0.0325130820274353, -0.5886677503585815, -1.522046446800232, 0.08815816044807434, -1.3303779363632202, 1.613223910331726, -1.1681770086288452, -0.8114581108093262, 0.31583061814308167, -0.39372411370277405, -0.32985299825668335, 1.8654981851577759, 1.0232999324798584, 0.47723397612571716, 0.26258668303489685, -0.48820167779922485, 1.4592772722244263, -0.18462027609348297, -1.5750577449798584, 0.5619246959686279, 0.9770955443382263, 0.7280286550521851, 3.2135653495788574, -1.8807979822158813, 0.7386162281036377, 1.1504794359207153, 1.4046574831008911, -0.47355541586875916, 0.9362383484840393, 0.34297072887420654, 1.0340932607650757, -0.9676405787467957, 0.8449319005012512, 2.0124692916870117, -0.41439613699913025, -0.15214715898036957, 0.7229722738265991, 0.36581185460090637, -0.913254976272583, 1.1839168071746826, -1.3168647289276123, -1.4385545253753662, -0.1002521887421608, -0.030369602143764496, 1.2458224296569824, 0.5829408168792725, 0.9489622116088867, 0.854682445526123, 1.7885938882827759, 0.46102118492126465, -0.14368948340415955, 0.7603351473808289, -0.28420889377593994, -0.1551172286272049, 1.3514858484268188, 1.9778047800064087, -0.9071383476257324, -0.488346129655838, 0.251010537147522, 0.42646047472953796, 0.28932419419288635, -0.35517653822898865, -0.7026726007461548, -0.3147751986980438, 0.4341560900211334, 0.5988650321960449, -0.30390647053718567, 2.2962019443511963, 0.04595085233449936, -0.10059873759746552, 0.7040190100669861, -1.0437344312667847, 1.0208144187927246, 0.022757912054657936, 1.5082759857177734, -1.3208568096160889, -0.6069127321243286, -0.19560830295085907, -0.724014163017273, 0.10583845525979996, 0.39817550778388977, -0.7052232027053833, 0.5683844089508057, 1.2940213680267334, -0.40886884927749634, 1.2380573749542236, 0.3158324360847473, 0.19325624406337738, -0.030564643442630768, -0.34390226006507874, 0.18308116495609283, 0.9616963863372803, -1.157678484916687, 0.815004825592041, -0.028551949188113213, 0.4245717227458954, 0.0815696194767952, 0.0709112137556076, 0.011733898892998695, -0.5553291440010071, -1.902935266494751, -2.13932728767395, 0.9181913137435913, -1.8431659936904907, 0.36814773082733154, 1.1984989643096924, -1.2775739431381226, 0.40121668577194214, -1.3872966766357422, -0.9008569717407227, -1.6791216135025024, 1.7844945192337036, -0.9330599308013916, -0.5787106156349182, -0.699511706829071, -1.166460394859314, 0.6582807302474976, 0.7056812644004822, 1.2636324167251587, -1.4623212814331055, 2.024390459060669, -0.5557284951210022, 0.1491413116455078, -0.8112814426422119, -0.12498240917921066, -0.6609019041061401, -0.17994877696037292, 0.040974169969558716, 0.3410045802593231, 0.6336386203765869, 1.1790156364440918, -1.2961945533752441, -0.3609287738800049, 0.025973984971642494, -1.1398695707321167, 0.33211401104927063, 0.49861258268356323, 0.028639232739806175, 0.6741936206817627, 0.6217085123062134, -0.5521705150604248, 1.0785815715789795, -0.09830786287784576, 0.10115741938352585, 0.3357667326927185, -0.1181630939245224, -0.5088589191436768, -0.05874419957399368, 1.1886175870895386, -0.15247680246829987, -1.0192111730575562, 0.10487648099660873, -0.4355491101741791, 0.10097817331552505, 1.2938696146011353, -1.9636790752410889, 1.3216851949691772, -1.539448857307434, -0.20859655737876892, -0.5482812523841858, 0.6523211002349854, -0.18519993126392365, 0.5363297462463379, -1.3201192617416382, -0.1811589002609253, -1.1480971574783325, -0.2603859603404999, -1.2477244138717651, 0.015320457518100739, 0.3128829598426819, -0.9343196749687195, 1.2567558288574219, 1.0144671201705933, -0.1664610207080841, -0.7704668641090393, -0.3754541575908661, -0.25497302412986755, -0.7616802453994751, 0.5690795183181763, -0.02530386671423912, -0.32376688718795776, 0.36808693408966064, 0.21972528100013733, 0.6353278756141663, -0.4667496681213379, 0.33725401759147644, -1.6816376447677612, -0.9196476340293884, -2.895538568496704, -0.5975083708763123, 0.2911519706249237, -0.10649462789297104, -1.476124882698059, -0.5290196537971497, 0.0868682861328125, -0.7409372329711914, -1.8257492780685425, 0.3665555417537689, -0.8340345025062561, 1.221483588218689, -0.3901212513446808, 0.28798311948776245, 0.5579692125320435, 0.8820793032646179, 0.015459992922842503, 0.05538458749651909, -0.5955378413200378, 0.3359379172325134, 1.602699875831604, -0.1299646943807602, -1.8830729722976685, 0.497546523809433, 0.8260316848754883, -0.9095130562782288, -0.638404369354248, 0.46950653195381165, -0.6594371199607849, 1.490149736404419, 0.0820469930768013, -0.9692929983139038, -0.08558841049671173, 0.41427716612815857, 0.754344642162323, 0.5738063454627991, -0.8430103659629822, -0.6363991498947144, 0.9532171487808228, -1.4717564582824707, -1.0481518507003784, 1.8218616247177124, -1.572839617729187, 0.04472329467535019, -0.42409372329711914, 1.0077605247497559, -0.17628000676631927, 1.023200511932373, -0.5809283256530762, 1.6945964097976685, -2.1935195922851562, 0.522765040397644, 0.23426760733127594, 0.2639845013618469, 0.16876345872879028, 0.08076933026313782, 1.372638463973999, 1.2795969247817993, 0.26273518800735474, -1.1654099225997925, -0.5506744980812073, 1.3696980476379395, 0.6917424201965332, 1.239721417427063, -0.607148289680481, -0.3300623595714569, -0.9847560524940491, -0.24332083761692047, -0.046469222754240036, -1.3312400579452515, -0.8677409291267395, 0.4613920748233795, 1.557922124862671, -0.06453206390142441, -1.5420684814453125, -0.9469243288040161, 0.08285349607467651, -0.8046050667762756, -1.7295905351638794, -0.40508532524108887, 1.4275859594345093, 0.8274089097976685, -1.3214013576507568, -0.7661513090133667, 0.6160814762115479, 1.2047315835952759, 0.8242798447608948, 0.14306500554084778, 1.3624626398086548, 1.7518690824508667, 0.49897414445877075, -0.33895084261894226, -1.209092378616333, 1.1509745121002197, -1.1139402389526367, -2.1854348182678223, 0.7900393009185791, 1.3563292026519775, -0.036387909203767776, -0.2515217959880829, -1.3631967306137085, -0.9525652527809143, 0.25264260172843933, 0.14761313796043396, -2.1840991973876953, 0.5533761978149414, -0.5056867003440857, 1.2965160608291626, 1.9520522356033325, 0.3000893294811249, 2.2411270141601562, -0.9549869894981384, 0.34639719128608704, 1.0067561864852905, 1.7951158285140991, 1.2815805673599243, -0.5482357740402222, 2.328587770462036, 0.43430382013320923, -0.17948386073112488, 0.7351452708244324, -1.4885286092758179, -0.7164940237998962, -0.05119234696030617, -0.9402266144752502, 0.7492842078208923, -0.4006204605102539, -0.8969570994377136, 1.916475534439087, -0.7418504953384399, -0.08754166960716248, -0.9019133448600769, 1.0276089906692505, 0.7048407196998596, 0.8181632161140442, 0.8035070300102234, 2.2568867206573486, 0.15662996470928192, 0.3379373252391815, -0.46508994698524475, -0.4858211576938629, 0.5822126865386963, -0.24185296893119812, -0.6363496780395508, -2.223031997680664, 0.7438108325004578, 2.031496524810791, -0.8162152767181396, -0.9389387965202332, -1.838813066482544, -0.1010088101029396, 2.33544659614563, -0.5303487777709961, -0.9457842111587524, -1.5823625326156616, 0.5710089206695557, 0.5295804142951965, -0.5889782905578613, -0.280497670173645, 0.25195589661598206, 1.7049630880355835, -0.02134350873529911, 1.563144564628601, 0.06544733047485352, -0.27986520528793335, -1.9762362241744995, -1.2595223188400269, 0.5805740356445312, 0.42493194341659546, 0.21039170026779175, 0.46918991208076477, 0.9214981198310852, -1.1426914930343628, 1.105897307395935, 1.7933087348937988, 0.630450427532196, 1.0283986330032349, -1.4352622032165527, -0.27320826053619385, 0.5942802429199219, -0.6872614622116089, -0.16528373956680298, -2.151261568069458, -0.5331997871398926, -0.25685805082321167, -0.10436145961284637, -1.5109202861785889, 0.2200598120689392, 0.20243163406848907, 2.0012640953063965, -0.2802727520465851, 0.8654574751853943, 0.4756103754043579, -1.3721064329147339, -0.7440496683120728, -1.7803499698638916, 0.7614084482192993, 0.135757178068161, 0.650735080242157, 0.4082874059677124, 0.01188420969992876, -1.3911556005477905, -0.7194713950157166, -0.41179215908050537, -0.2935722768306732, -1.1399400234222412, -0.07520241290330887, -1.1521543264389038, -0.18466435372829437, -0.17048852145671844, -0.22601518034934998, -0.23066823184490204, -0.9960601329803467, 0.51772141456604, -0.45111027359962463, -0.22596552968025208, -0.5214137434959412, -0.8323427438735962, 1.6666964292526245, 1.2808218002319336, -0.4631436765193939, 0.6302330493927002, 1.845509648323059, 0.4228535294532776, 1.5184799432754517, -0.060955096036195755, -0.8923082947731018, -0.2669554054737091, -0.15375201404094696, -0.14544764161109924, 0.767196536064148, -0.603326678276062, 0.06254575401544571, -1.4020439386367798, 0.03784436360001564, 0.7410939335823059, 0.18528158962726593, -0.5957891345024109, 1.7317456007003784, -0.25753819942474365, 0.5427194237709045, -1.471985936164856, 0.3029196262359619, 0.7290796041488647, -0.005063232034444809, 0.6711825728416443, 0.12833639979362488, 0.4676862061023712, -1.2848743200302124, 0.10242733359336853, -0.7714313268661499, 1.8199458122253418, 0.48014700412750244, -0.7280704975128174, 0.4387407898902893, -0.2941972613334656, 0.7324767112731934, 0.0984637588262558, 0.1350869983434677, 0.42173123359680176, -0.0552983321249485, -0.8338844776153564, -0.6792539954185486, -0.010720323771238327, 0.40347859263420105, 0.44848817586898804, 0.9392042756080627, 1.0233210325241089, -0.5761924982070923, -1.4884650707244873, -0.18250365555286407, 1.3550786972045898, 1.8759374618530273, -1.0036190748214722, -0.6086689233779907, -0.4013301432132721, -1.8780618906021118, 0.8022327423095703, -1.470602035522461, -1.4158302545547485, 0.3759307861328125, 1.6678004264831543, -0.9066570997238159, -0.6410021781921387, -1.167093276977539, 0.617125391960144, 0.8113125562667847, 0.9086247682571411, 0.38387978076934814, 0.6251119375228882, 0.4908755421638489, 0.054989855736494064, 0.2101728618144989, 0.4064839482307434, 1.1416136026382446, -0.7502548098564148, 0.011711375787854195, -0.8628848791122437, -0.5086080431938171, -2.137838363647461, -1.3137080669403076, -1.7595372200012207, -0.9004535675048828, 1.005602478981018, -0.3902292251586914, 1.1020830869674683, 0.7668673992156982, -0.7872450351715088, 0.7748846411705017, 2.610382080078125, -0.19470785558223724, -0.8484358191490173, -1.461308479309082, 0.08354270458221436, 0.7983909249305725, -0.24765069782733917, 0.9764372706413269, 0.02418016456067562, -0.8492945432662964, -0.7686991691589355, -0.9287771582603455, 1.768000602722168, 0.19096529483795166, -0.30195149779319763, 1.6531251668930054, 1.2529430389404297, 0.15396659076213837, -0.8318083882331848, -0.1318451315164566, -0.9668772220611572, 0.7099048495292664, -1.3495782613754272, -0.31531092524528503, 1.8569117784500122, -0.3821828067302704, 0.5308864712715149, -0.6867265701293945, 1.5096454620361328, 0.07734028249979019, 0.6392923593521118, -0.33246058225631714, 2.505047559738159, 2.054384231567383, -0.573371946811676, -0.5329919457435608, -0.7516852617263794, -0.9057583212852478, 0.8188066482543945, 0.09646140038967133, -1.2372653484344482, 0.35999616980552673, -1.2086050510406494, 0.02101529948413372, -0.25408196449279785, -0.403110533952713, 0.588659942150116, 2.664990186691284, -0.09743988513946533, 0.10672669112682343, 0.6824946999549866, -0.39899322390556335, 2.524852991104126, 0.8113119006156921, -1.4108151197433472, 0.3336533308029175, 0.48661836981773376, -0.6527165174484253, 0.6392375826835632, -0.5078438520431519, -0.4103226661682129, 2.3242433071136475, 2.895800828933716, -0.8857381939888, -0.7014948725700378, 1.1612075567245483, -1.863032341003418, -0.24873951077461243, -0.24216853082180023, -1.071797490119934, -0.3477433919906616, -1.1928024291992188, -1.2311428785324097, -0.8654888868331909, -1.6319119930267334, -1.2735273838043213, 0.8767536282539368, 0.06155145913362503, 0.41067957878112793, 2.2189881801605225, -0.20760199427604675, -0.8913484811782837, 1.5623358488082886, 0.10245633870363235, -1.393675446510315, 1.32620370388031, -0.9895116090774536, 0.7529857754707336, 0.4208926856517792, -1.3039991855621338, -1.4671076536178589, 1.7838258743286133, 1.7724846601486206, -0.10020039975643158, -0.5067446827888489, -0.6800538301467896, 2.917572259902954, -1.7605338096618652, 0.21868695318698883, -0.45889800786972046, 1.0018181800842285, 0.07676007598638535, -0.6175537109375, 1.3598904609680176, -1.0637521743774414, 1.6285951137542725, -0.5761444568634033, 0.3519595265388489, 1.1331309080123901, -0.860318124294281, -0.8843072652816772, 0.8697404861450195, 0.5133914351463318, -0.8164416551589966, 0.3048933148384094, -0.7467212677001953, 1.1851063966751099, 0.9244225025177002, 0.3240814507007599, -0.19758182764053345, -0.2026480883359909, 0.20646053552627563, -1.6835761070251465, -1.0183523893356323, 0.8695803880691528, -1.1041276454925537, 0.6397362947463989, 0.23038151860237122, -1.4351204633712769, 0.17453114688396454, -0.6202817559242249, 0.6715859174728394, -0.44446802139282227, -0.4423264265060425, 0.6732156872749329, 0.05946206673979759, -1.0410255193710327, 0.09712059795856476, 0.95946204662323, -0.06709016859531403, -0.08119342476129532, 0.8622798919677734, 0.03330226615071297, -0.7092985510826111, -1.111172080039978, 0.4380459487438202, -1.4166021347045898, -0.25058597326278687, -0.6284664273262024, -1.3723348379135132, -0.6025580763816833, -1.0129354000091553, -0.41081809997558594, 0.3836456835269928, -0.7292227745056152, 1.638350248336792, -1.502976655960083, 3.023012161254883, 1.4689133167266846, -0.1834791749715805, -1.6494919061660767, -0.4429214894771576, -2.0275979042053223, 0.15102854371070862, -1.8450099229812622, -0.3932134211063385, -0.4274226129055023, -0.5634457468986511, -0.28756439685821533, 1.0777226686477661, 0.5387145280838013, 1.2550400495529175, 2.1980886459350586, 0.0545954704284668, 0.035501591861248016, -0.15436862409114838, -1.1869803667068481, -0.6262980699539185, -2.5702502727508545, -0.40185272693634033, -0.2635030448436737, -0.35784924030303955, -1.0207902193069458, -0.1971961110830307, -1.6686376333236694, 0.4286004304885864, 0.860210657119751, -0.4238715171813965, -0.8155635595321655, 1.9433047771453857, 1.5722659826278687, 0.5208386182785034, -0.8672803044319153, -1.3764890432357788, 0.34027910232543945, 0.6802203059196472, 0.32881709933280945, -0.17322422564029694, -0.3518144190311432, 0.3679491877555847, 0.8030157685279846, -1.8342310190200806, -0.31844422221183777, -2.053938865661621, 0.9990371465682983, 0.6821284890174866, 0.707934558391571, -0.1855129450559616, 0.2531181871891022, -0.14490853250026703, -0.44566109776496887, -0.5712705850601196, -0.17127259075641632, -0.009212939068675041, -1.3424925804138184, -0.07285016775131226, 0.3796168863773346, -0.20432822406291962, 0.3621923327445984, -0.2404167801141739, -0.34648820757865906, 0.33828604221343994, -1.376211404800415, 0.0395594984292984, -0.7161042094230652, 1.5061323642730713, -0.13489767909049988, -0.5902533531188965, -0.11264341324567795, -0.1402229219675064, 1.0839757919311523, 0.5197489857673645, -0.03641759604215622, -0.9061479568481445, -0.6317787766456604, 0.08555673062801361, 1.5084238052368164, 0.4548306167125702, 0.050610341131687164, 2.3359146118164062, 1.9224145412445068, -1.888384461402893, -2.2655677795410156, -0.8225789666175842, -0.3986349403858185, -0.39706194400787354, 0.8331148624420166, 1.1734607219696045, 0.602148175239563, -0.47069230675697327, -0.6245542168617249, -0.03582808002829552, -0.5365073680877686, -0.34665045142173767, -0.6879498362541199, 0.996691882610321, -0.6742205023765564, 0.7049444913864136, -0.9266670346260071, 0.9587987065315247, -0.2292250394821167, -0.5827191472053528, 1.2608345746994019, -0.320891410112381, 0.6173056364059448, -0.332242876291275, -1.6040514707565308, 0.7995389103889465, 1.0119349956512451, -2.984873056411743, 0.0735616609454155, -0.26015526056289673, -0.45339435338974, 0.9548648595809937, -1.5505200624465942, -0.20554745197296143, -0.14504793286323547, -1.1056708097457886, 1.3918412923812866, -1.019932746887207, -0.04349195212125778, -1.2886533737182617, -0.48798978328704834, -0.48562175035476685, 0.22060634195804596, 0.655620813369751, 0.4055013954639435, 0.307858943939209, 0.005887818522751331, -0.5444408059120178, -1.344391942024231, -1.8956058025360107, -0.7685821652412415, -0.7793465256690979, 0.40112003684043884, -0.676416277885437, 0.8589715957641602, 0.5759567618370056, 0.42178577184677124, -0.43001335859298706, 0.23212842643260956, 1.5184890031814575, 1.3280010223388672, 0.6765753626823425, 0.517204761505127, -1.1156033277511597, 0.28448978066444397, 0.7636358141899109, -0.6127161383628845, -0.2762391269207001, -0.5929362773895264, -0.34862279891967773, -0.8299728035926819, 0.22901883721351624, -0.5893693566322327, -0.18225568532943726, -0.3536534011363983, 1.1443204879760742, -0.23808617889881134, 2.296644687652588, -0.5873817801475525, 0.35106033086776733, -0.9726987481117249, -0.0529879555106163, 1.6578956842422485, -0.9100603461265564, 0.13013914227485657, 0.19023671746253967, 0.9986467361450195, -0.15785972774028778, 0.5907794237136841, -0.18442101776599884, 0.22296969592571259, 0.3246020972728729, 0.8789464831352234, 0.2871592938899994, -0.8194026947021484, 0.2921368181705475, 1.5402389764785767, -1.039255976676941, 0.5679640173912048, -0.21044355630874634, 0.8391956090927124, 0.8556926846504211, 0.5300695896148682, -0.7310124635696411, -0.015835126861929893, -0.3682985305786133, -1.5478193759918213, -0.19919955730438232, 0.7417209148406982, 1.0915753841400146, 0.46754947304725647, -0.2314988374710083, 1.4174139499664307, -1.672201156616211, 2.071385383605957, -0.5881354212760925, 0.519519567489624, -0.5913510322570801, -0.803390622138977, 0.08944819122552872, 1.6062601804733276, 1.3554238080978394, -0.2501862645149231, 0.6387078166007996, -0.03676187992095947, 0.49649420380592346, -0.3104080259799957, -0.29357385635375977, 0.10889190435409546, -0.8913748860359192, -0.7272117137908936, 0.07769922912120819, 0.23940277099609375, -0.28313007950782776, 0.5526116490364075, -0.2559742033481598, 0.6916399598121643, 0.35949936509132385, -0.4079248905181885, -0.27819886803627014, 2.1977686882019043, -2.4620585441589355, -0.1664830446243286, 0.12068304419517517, 0.1483432948589325, 0.8186697363853455, 1.0508028268814087, 0.4947822690010071, 0.7463624477386475, 0.22655127942562103, -1.5028119087219238, -0.730455219745636, -1.224875569343567, 1.404188871383667, 0.48435279726982117, -0.06489235907793045, 1.0838788747787476, 0.09926192462444305, -0.3613237738609314, -0.03103584609925747, -0.5185584425926208, -0.36074432730674744, 0.5287497639656067, 1.3063640594482422, 1.6558095216751099, 1.0503509044647217, -0.14539633691310883, 1.150609016418457, -0.64842689037323, -0.3767986595630646, 0.6752933263778687, -1.0812665224075317, 0.3919593393802643], [1.1635209321975708, 0.14619919657707214, 0.14819060266017914, 0.361965149641037, -0.8022318482398987, 0.2165766805410385, 0.4409905970096588, 0.502311110496521, -0.35422852635383606, 0.4941920042037964, 1.4023869037628174, 0.683729887008667, -1.4075512886047363, 1.365930438041687, 1.6205390691757202, 0.05205276235938072, 1.1634668111801147, 0.5669211149215698, -0.8892083168029785, -0.370822012424469, 0.7649350166320801, -1.4527872800827026, 0.5811544060707092, 1.1416912078857422, 0.8464603424072266, 0.0043785753659904, -0.6800857186317444, -0.5860164165496826, 1.1526472568511963, -0.4763158857822418, -0.44068607687950134, 1.24819815158844, -0.47616004943847656, -0.0221246387809515, -1.5392554998397827, -0.41975849866867065, -1.0572896003723145, -0.8275836110115051, -0.1266188621520996, -0.27248409390449524, 0.15796148777008057, -1.482865571975708, -0.10330532491207123, -1.6727092266082764, 1.5668914318084717, -1.5225166082382202, -0.439766526222229, -0.20468275249004364, -0.09552387148141861, -0.7343164086341858, -0.3239821195602417, 1.5463640689849854, -0.47694242000579834, 0.6902217268943787, 0.5762506723403931, -0.010010977275669575, -0.017251737415790558, -1.0922629833221436, 0.9641368985176086, -0.048509228974580765, -0.562214732170105, -0.7733303308486938, -0.6239671111106873, -0.7021698951721191, -0.17546778917312622, 0.5532370209693909, 1.1626667976379395, -0.7105023860931396, -0.35293468832969666, 0.34444278478622437, -1.2177175283432007, -0.7538335919380188, -0.9904400706291199, -0.3147571384906769, -0.26075950264930725, -1.614567518234253, 1.7440129518508911, 0.6290034651756287, 0.6517756581306458, -1.0793743133544922, -2.640536069869995, 1.4927618503570557, -0.45188701152801514, -2.130370855331421, 0.7379221320152283, -0.4938751459121704, 0.1301141232252121, 0.8803095817565918, 0.40233469009399414, 1.9543471336364746, 0.18737603724002838, 0.7533558011054993, 1.8820635080337524, -0.3372129201889038, -0.7860177159309387, -0.2146303802728653, -0.40595224499702454, -1.5326240062713623, -0.69217449426651, -1.902171015739441, -0.263359934091568, -1.547692894935608, 1.5946896076202393, -1.1312758922576904, -1.4059430360794067, 1.1632596254348755, 0.3277277946472168, -1.0603787899017334, -1.8393571376800537, 1.2007521390914917, 1.2283204793930054, -0.5321767926216125, 2.271409511566162, -0.04667787626385689, 0.1616702526807785, -0.5878838896751404, 0.37577909231185913, -0.052834007889032364, 0.6393346786499023, 1.7363715171813965, -0.7999892234802246, 2.1562633514404297, -0.4346464276313782, -0.2853655219078064, 0.3912968933582306, -2.1564526557922363, -0.8435148000717163, -0.1507984846830368, 1.4750609397888184, -0.10968849062919617, 0.1422257423400879, -1.963175654411316, -0.46732452511787415, -1.369979977607727, 0.3968762159347534, 1.330126166343689, -0.3430846631526947, -0.07121135294437408, 1.2882399559020996, 1.37562894821167, -0.504034698009491, 0.907522439956665, 0.7983997464179993, 1.7127436399459839, -0.5999582409858704, 0.9231492877006531, -0.1673078089952469, 0.6694959998130798, -0.702991247177124, -0.9227761030197144, 0.44318461418151855, 0.7327730655670166, 0.9268336892127991, 0.39039045572280884, -0.014132427982985973, -0.41263049840927124, 0.568341851234436, -0.4157869517803192, -1.6741708517074585, 0.7698623538017273, -0.4956050217151642, 0.03407759964466095, -0.19857743382453918, -1.5767170190811157, 0.0838170126080513, 1.6024501323699951, 1.2616783380508423, -0.5569775700569153, -0.13764828443527222, -0.6437827944755554, -0.951304018497467, -0.7750086784362793, -1.1461676359176636, -0.052187029272317886, 0.5620593428611755, -0.9982212781906128, -0.7421244382858276, -0.7204259037971497, 0.26739501953125, -1.2770658731460571, -0.5654839873313904, 0.39037343859672546, -1.170575737953186, 0.6492361426353455, 0.6168704628944397, 0.5951554775238037, 0.857261061668396, -2.222273349761963, -0.17423462867736816, 0.38717925548553467, 0.7156057357788086, -0.27202263474464417, 0.8159262537956238, 0.6281788945198059, 0.7349134087562561, -0.025543108582496643, 1.2177995443344116, 1.6213405132293701, -0.7667315006256104, 0.6574071645736694, 0.3553408682346344, -0.16278578341007233, 0.0452539436519146, -0.7652885913848877, 0.9816315770149231, -0.28286609053611755, 0.6871593594551086, 0.5540499687194824, 1.0752781629562378, -0.4355560541152954, 1.0027434825897217, -1.355065107345581, -0.38719508051872253, -0.1304226964712143, -0.0800611823797226, -0.8667450547218323, 0.4934338331222534, 0.2727230489253998, -0.19732485711574554, -0.8829214572906494, 0.931837260723114, -0.7569766044616699, -1.0701528787612915, -0.948899507522583, -0.13117867708206177, -0.2976846694946289, -0.7390097379684448, 0.5326383709907532, 1.4445478916168213, -0.3562176823616028, -0.35975486040115356, -1.9905391931533813, 0.7580679655075073, 1.036007046699524, -0.43604573607444763, -0.008848929777741432, 1.7066919803619385, -0.3013356328010559, -0.0417286641895771, -0.32416626811027527, 2.2817888259887695, -0.6282898783683777, -1.8203394412994385, 0.5104719996452332, 0.6202080845832825, 0.19823415577411652, 0.9817327260971069, 0.5667790174484253, -0.51569664478302, 1.6014890670776367, -0.2791701853275299, -1.5614335536956787, -0.1995547115802765, -0.1142735481262207, -0.6301331520080566, -0.9904428124427795, 0.5337567329406738, 0.014253259636461735, 0.5333322286605835, 0.3532492518424988, 1.1331584453582764, -0.7282333374023438, -1.9674605131149292, -0.6123667359352112, -1.2912362813949585, 0.7100985050201416, -1.5923662185668945, 2.3759119510650635, 0.3735150396823883, 0.5182023644447327, -0.2965625524520874, -0.24732093513011932, 0.9270935654640198, 1.1533231735229492, -0.21389828622341156, -0.7373613715171814, -1.5972880125045776, -0.45496121048927307, 0.7745370268821716, 0.8866919875144958, 0.12165721505880356, -0.018434137105941772, 0.9292005896568298, 0.727616012096405, -0.2547959089279175, -0.5911734104156494, 0.6345806121826172, -0.5698757171630859, -0.9701248407363892, -0.5759088397026062, -1.2475947141647339, -0.42086344957351685, 0.9764580726623535, 1.4269461631774902, -0.805846631526947, 1.0549577474594116, -1.4394071102142334, -0.9231400489807129, 0.09669936448335648, 0.5836694836616516, -1.9779633283615112, -0.14311985671520233, -0.4561072587966919, -0.5233736038208008, -2.5342843532562256, -1.038024663925171, -0.6699457764625549, 1.2879643440246582, 0.9863489270210266, -1.0180262327194214, -0.823076069355011, -0.4742356836795807, -1.3253999948501587, 1.971267580986023, -1.5008822679519653, -0.07378862798213959, 1.3662189245224, -0.2396681159734726, -0.5929818153381348, 0.6256716251373291, -0.6918535232543945, -0.9163339138031006, 0.14070568978786469, 0.7345240712165833, 1.8061208724975586, 0.25867578387260437, -0.7810766100883484, -1.2780399322509766, -0.44767412543296814, 2.5875401496887207, -0.7038517594337463, 0.38827309012413025, 0.8966336846351624, -1.0828603506088257, -0.5957357883453369, -0.8819109201431274, 1.5621174573898315, 0.9332594871520996, 0.08948215842247009, -0.9335442781448364, -1.0683715343475342, -0.8209963440895081, -1.0044766664505005, -0.8259633779525757, 0.6944952607154846, 0.3909786343574524, -0.1094493567943573, -1.364354133605957, -0.03770624101161957, -0.31530553102493286, 0.39801251888275146, 0.27111923694610596, -0.9908389449119568, 1.321738600730896, -0.5929373502731323, 0.145859956741333, 2.044898509979248, 0.7409523725509644, -0.4344562590122223, -0.39723944664001465, -0.15025107562541962, 2.2621009349823, 0.6783785820007324, 1.4366952180862427, 0.7224514484405518, -1.3500601053237915, -1.494424819946289, -1.0406831502914429, 0.24224120378494263, 0.864450991153717, -0.5384911298751831, 0.22217027842998505, 0.22784806787967682, -0.49780577421188354, -0.06497248262166977, -1.5654730796813965, -0.7380357980728149, -0.39303576946258545, -0.5833257436752319, -0.8384091258049011, 0.9741315841674805, -1.6082477569580078, 0.611701488494873, -0.7016205191612244, 1.3466852903366089, -1.7009774446487427, -0.2601616084575653, -0.055119093507528305, 0.5876769423484802, 0.26356613636016846, 0.11872650682926178, -0.8000355362892151, -0.1758521944284439, -0.4880326986312866, -0.48723161220550537, 0.38547492027282715, -1.4266140460968018, -1.330318808555603, -0.18823128938674927, -1.5111271142959595, -0.711721658706665, 0.32593652606010437, -1.5440768003463745, -0.504305362701416, -0.563147246837616, 0.4450184404850006, 0.5842351913452148, 0.13519637286663055, -1.0053969621658325, -1.125002384185791, -1.2357860803604126, 0.8669983744621277, -0.08554738014936447, -0.7226868271827698, 0.9744652509689331, 1.5231374502182007, -0.04000186175107956, -0.35287556052207947, -0.22498390078544617, -0.14095288515090942, -1.2164911031723022, 0.5205329060554504, 0.1232096254825592, 0.12522949278354645, -1.134079933166504, 1.70650053024292, -1.3817343711853027, -0.14137928187847137, 0.46608787775039673, 0.6534671783447266, -0.28489527106285095, -0.022794675081968307, 0.4806601405143738, -0.5531495213508606, -1.3547662496566772, -0.25617551803588867, 2.9877359867095947, 0.3382774889469147, -1.4621587991714478, 0.9455613493919373, 0.5768886208534241, 0.7220835089683533, 0.5749118328094482, 0.7589380741119385, -0.28832489252090454, 1.0926570892333984, -1.2262660264968872, -0.021201524883508682, -0.160589799284935, -1.0893948078155518, -0.566511869430542, -0.6394826769828796, -0.9828774929046631, 0.861090898513794, -1.1545523405075073, 0.2534714639186859, -0.0807734802365303, 1.2652491331100464, -0.8576374650001526, -1.1971274614334106, 0.5432420372962952, -0.4210180938243866, 2.1542258262634277, -0.8333889245986938, 0.1935116946697235, -0.20796287059783936, 1.6455668210983276, -1.6950114965438843, 0.45124852657318115, -1.1659111976623535, 0.2885473966598511, 0.5310738682746887, 1.0347843170166016, 0.5670681595802307, -2.279330253601074, -0.669116735458374, -0.9855743050575256, 0.16341198980808258, -0.13654187321662903, -0.45486608147621155, -0.3275821805000305, -0.05419463291764259, 0.2035326212644577, 0.3897753953933716, 2.1444170475006104, -1.601128339767456, 1.3352270126342773, 2.045882225036621, 0.11939211934804916, -1.4609845876693726, 0.7310936450958252, -1.4679982662200928, -0.2935359477996826, 1.0949186086654663, 0.13240894675254822, -0.6627548933029175, 1.095307469367981, 1.6589202880859375, 0.8316748738288879, 2.1054141521453857, -0.8566089272499084, -0.9053725004196167, -0.28381112217903137, -0.6837829947471619, -2.8638479709625244, 0.19383515417575836, 0.301089346408844, -0.948083758354187, 0.12061847001314163, -0.6915379166603088, 0.1890309453010559, -0.17890219390392303, -0.03234543651342392, 1.0006134510040283, 1.3165836334228516, 0.037790447473526, -1.7718905210494995, -1.4807885885238647, -0.5063551068305969, 0.4948868453502655, 1.4246296882629395, -0.7770153284072876, 0.854739248752594, 2.0869643688201904, -0.41259485483169556, -0.8312525153160095, 2.0435869693756104, 1.3646678924560547, 0.44314491748809814, 0.7480179667472839, -0.5209149718284607, 0.7180229425430298, 2.1741511821746826, 2.2159855365753174, 1.4469561576843262, -0.9474064707756042, 0.6734805107116699, -0.6261420249938965, 0.6937596797943115, -1.417401909828186, -1.046198844909668, 0.8896880149841309, 0.24728883802890778, 0.1683460772037506, -1.1827239990234375, -0.46980974078178406, -1.1951621770858765, -0.4601343274116516, -0.5151997804641724, -2.544226884841919, 0.1305093616247177, -0.08715986460447311, -1.734518051147461, 1.2415930032730103, 1.288780689239502, -0.3620149493217468, 1.6532336473464966, 0.5113905072212219, 0.9314450025558472, -1.0092847347259521, 0.4970669448375702, -1.9873172044754028, -0.18218883872032166, -0.31576070189476013, 1.5418065786361694, 2.148871898651123, -1.2884453535079956, 0.5859937071800232, -0.976199209690094, -0.3367713987827301, 1.06959867477417, 0.4935484826564789, -0.7080455422401428, 0.10597721487283707, 0.416128009557724, -1.3308894634246826, -2.16618275642395, -0.7554555535316467, -1.7027530670166016, 0.2163405567407608, 1.2000372409820557, 0.8721571564674377, -0.04986685886979103, -0.9271848201751709, -2.043761730194092, 1.1782760620117188, 0.34326744079589844, -0.6600497961044312, 0.8134374618530273, -0.10748082399368286, 0.32223275303840637, -0.5190662741661072, 0.22821080684661865, -0.27455613017082214, 0.7558382153511047, 1.5561916828155518, 0.8208802342414856, 0.29019439220428467, 0.2487168312072754, -0.08757606893777847, -0.7773325443267822, -0.8676052689552307, 1.3754286766052246, -0.01550732459872961, 0.14962638914585114, 0.2983037233352661, -0.6869369745254517, -0.9046936631202698, -1.2399485111236572, 0.4680595099925995, 1.8000901937484741, 0.9036475419998169, 0.15540769696235657, 0.9559175968170166, 0.8266558647155762, -0.3784217834472656, -2.5744760036468506, -1.2429078817367554, -0.5465337038040161, -0.5724977254867554, -1.3081815242767334, -0.47141319513320923, 0.7907958626747131, -1.3220372200012207, -0.38328951597213745, 0.8113382458686829, -0.5583121180534363, 0.17326876521110535, 1.437170147895813, -0.29947832226753235, -0.9158334732055664, 0.10827809572219849, 0.7237501740455627, 0.4378400146961212, 0.3975287973880768, 0.8789612650871277, 0.06850705295801163, 0.029071617871522903, 0.08774913102388382, -1.0163358449935913, 0.24169903993606567, 0.9091981053352356, 0.16053085029125214, -0.44943058490753174, -1.658373236656189, 2.0091731548309326, -2.341099739074707, -0.34685656428337097, -0.5507888793945312, 0.44377028942108154, -0.8483453392982483, -0.9579114317893982, 1.095700740814209, 0.5222547054290771, 0.1888524889945984, 1.5072832107543945, 0.9721004962921143, -0.9543978571891785, -0.4580576717853546, -0.5556376576423645, -0.22927026450634003, 1.0676872730255127, 0.13716085255146027, 2.3924903869628906, -0.28238922357559204, 0.6844319701194763, -1.1747146844863892, 0.17553795874118805, -1.2965373992919922, 0.6521512269973755, -0.20119255781173706, 0.7543437480926514, 0.40496936440467834, -0.5059919357299805, -0.3775019645690918, -0.41343405842781067, 0.3524211347103119, -0.119805708527565, 1.598593831062317, 1.3822327852249146, -0.19415469467639923, -0.32109498977661133, 0.9334579706192017, 0.14336340129375458, 0.06628973037004471, -0.6755691170692444, -1.0957081317901611, -0.0054617635905742645, 1.401437759399414, 0.23062340915203094, -0.31440022587776184, -0.4973302185535431, -0.7032362818717957, -0.5687089562416077, -1.6304512023925781, 0.14835356175899506, -2.1919102668762207, 0.273359477519989, -0.014152918010950089, -0.798937976360321, -0.6864361763000488, -0.27448034286499023, -0.4594011604785919, -3.1952965259552, 1.0418657064437866, 0.5155580639839172, -0.8822129368782043, -0.8790731430053711, -1.247999906539917, -1.2490280866622925, 0.24518029391765594, -2.926650047302246, 0.07601210474967957, 0.18201503157615662, -0.7756534814834595, 0.18553458154201508, 0.6374116539955139, 0.8465114831924438, -0.022088011726737022, 0.05034340173006058, 0.404269278049469, -0.9029369354248047, 1.1698238849639893, -0.8132707476615906, -1.142943024635315, -1.195208191871643, 0.28604263067245483, 0.0683681070804596, -0.861914336681366, -0.24433201551437378, -1.0614370107650757, 0.9922587871551514, 0.8231953382492065, 1.2309612035751343, 0.2964731752872467, 0.3329926133155823, -1.1777143478393555, -2.0885009765625, 0.5778929591178894, -0.7224347591400146, -0.7918314337730408, -0.09093617647886276, 0.26345720887184143, 0.0009572473936714232, 0.3975100517272949, -1.0469517707824707, 0.4720720946788788, 0.545150876045227, -1.7182855606079102, -2.1255626678466797, 0.5271353125572205, -0.6743553280830383, -0.4280475080013275, -1.4356544017791748, 1.078160285949707, -1.3287447690963745, -0.51351398229599, 0.4704018831253052, -1.34647536277771, -0.3680337369441986, -0.5226370692253113, -0.3628409504890442, 0.4413837194442749, 2.557349920272827, -2.0390214920043945, -0.7573664784431458, 1.9206300973892212, 0.33468151092529297, -0.4187818169593811, 0.0279039666056633, 2.1281139850616455, 1.1571139097213745, -0.09157547354698181, 1.6252683401107788, -0.8871286511421204, -0.7558712959289551, 2.557950735092163, -0.07703792303800583, 0.8834259510040283, 0.5337148308753967, -0.1340567171573639, -1.7096935510635376, -0.289825975894928, -0.045523501932621, -1.384748101234436, 0.07247724384069443, 0.40920549631118774, 2.112426280975342, 0.6192889213562012, 0.6725805997848511, -2.339782476425171, 1.005798101425171, -0.34255251288414, 1.314368724822998, -0.19313593208789825, 0.36644548177719116, 0.7024115324020386, 0.2831667959690094, -0.9934471249580383, -1.370748519897461, -0.16559982299804688, 0.25502148270606995, 1.360499382019043, 0.8599662780761719, -0.5662202835083008, -1.4299732446670532, -1.146497368812561, -0.607363760471344, -0.9352027177810669, 1.1077284812927246, 0.9301069378852844, -1.675025463104248, -0.12942546606063843, -1.2889269590377808, -1.4459850788116455, -1.8224310874938965, 0.7831022143363953, -0.21653950214385986, 0.6997743248939514, 1.0714308023452759, -0.3221771717071533, 0.29778262972831726, 0.3444364070892334, 2.3830878734588623, 1.5739350318908691, -2.3615078926086426, 0.9818309545516968, 0.897375762462616, -0.8158373832702637, -0.4717460870742798, -0.46280255913734436, 0.7143493890762329, -0.1823982298374176, -0.014153482392430305, -0.1602054387331009, 0.824744462966919, -0.7944907546043396, 0.2808625102043152, -0.5602651238441467, 0.835873007774353, 1.7530182600021362, -0.9219508767127991, -0.19891375303268433, 0.5514221787452698, 1.3304749727249146, 0.8291770219802856, -0.04941565543413162, 0.9831224679946899, -0.3748200535774231, -0.292367160320282, 0.061515603214502335, 1.4945636987686157, -0.4742918908596039, 0.10887639224529266, 1.5174808502197266, -0.4461689293384552, -1.0763362646102905, 0.026592863723635674, 0.11579575389623642, -0.3653552830219269, -1.5383745431900024, 0.8462042212486267, -0.11544634401798248, 0.24776285886764526, 0.8025081157684326, 0.05678357556462288, 0.36419859528541565, 0.39058566093444824, 1.0399171113967896, 0.9424943923950195, 0.7362246513366699, 0.043583303689956665, -0.45907431840896606, -0.04333701357245445, 0.8676222562789917, -0.3758585453033447, 0.37365278601646423, -1.074202537536621, -0.8879030346870422, 1.4780831336975098, -0.49208223819732666, -0.41365447640419006, -2.0124971866607666, 0.5700539946556091, 0.7322820425033569, 2.018569231033325, -0.2665718197822571, -0.6164975762367249, -2.0354321002960205, 0.06813579797744751, -0.6984853148460388, -0.2665874660015106, 1.0643529891967773, 1.7417575120925903, -0.7263658046722412, 0.5112078189849854, 0.798987090587616, 2.162454605102539, -0.5683426856994629, 0.1536034494638443, -1.9744278192520142, 0.5632345676422119, 0.5413276553153992, 0.14319275319576263, 1.0531214475631714, -0.41070112586021423, -0.9265990257263184, 0.1079585999250412, 0.39000681042671204, 0.05992071330547333, -1.4757814407348633, 1.1109822988510132, -1.376244306564331, 1.1115671396255493, 1.147830843925476, -1.1729949712753296, -0.7390843629837036, -0.8849379420280457, -0.028383636847138405, 1.6424410343170166, 0.43341419100761414, 0.8586639761924744, 0.2388135939836502, 0.962314784526825, -0.20173966884613037, 0.36384522914886475, 1.7501611709594727, -0.5843713879585266, 0.43541333079338074, -1.272420048713684, 1.2264927625656128, 0.060788992792367935, -0.27783575654029846, 0.48853203654289246, 0.5772230625152588, 0.4925437271595001, 0.651213526725769, 0.09188520908355713, -1.0355384349822998, 1.2839410305023193, -0.5454861521720886, -0.3058370053768158, -1.4619112014770508, -0.9456309676170349, -0.8722257614135742, 1.2080914974212646, 0.7339491248130798, 0.18264873325824738, -0.042602911591529846, -0.17188476026058197, 0.840453028678894, 0.9192450046539307, 1.5190924406051636, 1.0395177602767944, -0.05668434873223305, -0.6944825649261475, 0.6779723763465881, -0.07327721267938614, 0.05770722031593323, 0.819796621799469, 1.1494203805923462, -0.21865621209144592, -0.6926999688148499, -0.04979715123772621, 0.6804160475730896, 0.8478060960769653, 0.1052887812256813, 1.715318202972412, 1.679262638092041, 1.0829744338989258, -0.13244189321994781, 1.0414525270462036, -0.6586114168167114, 1.1806423664093018, 0.8997883796691895, 0.3759075105190277, -1.8767180442810059, -1.9197137355804443, 0.7589598894119263, 1.3054109811782837, -0.22556741535663605, -1.0959153175354004, 0.35856717824935913, -0.36174139380455017, -0.5482457280158997, -0.8916946053504944, 1.4188882112503052, -0.9127044081687927, -0.3226541578769684, -1.9695773124694824, 0.2883569002151489, 0.5040608048439026, 0.9581636190414429, 1.5320734977722168, 0.021753229200839996, 0.4436938762664795, -0.22974063456058502, 1.4823813438415527, 0.33726799488067627, 0.14784739911556244, -0.9432528018951416, -0.49583205580711365, -0.755363404750824, -0.22396840155124664, -0.20344577729701996, 1.0444772243499756, -0.3261226713657379, 0.6286880373954773, -1.022830605506897, 1.302394986152649, -1.1451044082641602, -0.7739748954772949, 1.0813401937484741, 0.4437405467033386, 0.13540998101234436, -0.7119130492210388, 0.7717475295066833, 0.9017732739448547, 1.6488571166992188, 0.44632554054260254, -1.5475741624832153, -1.593501329421997, 0.6438639163970947, -0.2421136200428009], [2.1691393852233887, -0.005063942167907953, -0.22359196841716766, -0.30712050199508667, 1.6996229887008667, 0.09973437339067459, -1.089236855506897, -0.21274961531162262, -0.7177794575691223, -0.7531780004501343, -0.9325125813484192, 0.00023461849195882678, -0.5366463661193848, -0.5158815979957581, 0.30930212140083313, 0.620154082775116, -0.5101615786552429, 0.05427790805697441, -0.5311664342880249, -0.06522488594055176, 1.2258899211883545, 0.998810887336731, 1.4438668489456177, -0.3939948081970215, -0.7575318813323975, 0.578855037689209, -0.44530409574508667, 0.880503237247467, 0.3337392508983612, -1.6770631074905396, 1.503749966621399, -0.6805000901222229, -0.3953334093093872, -0.6560719609260559, 0.5697153806686401, 0.5558876991271973, -1.1930934190750122, -0.05287152901291847, -0.1280323714017868, 0.822842001914978, -0.4522550702095032, 0.10175837576389313, -0.1378113031387329, -0.015570088289678097, 0.14735370874404907, 0.35086432099342346, -0.41583016514778137, -1.372236728668213, 0.09362412989139557, -0.37639203667640686, -0.575361967086792, 0.9627715349197388, 0.7484585046768188, 0.6011264324188232, -0.4991983473300934, -0.627045750617981, 0.7951515316963196, 1.1014645099639893, 1.3183634281158447, 0.43309304118156433, 0.15733322501182556, 0.03450261056423187, 0.848894476890564, 2.077915668487549, -0.9947090148925781, -0.22250838577747345, 0.6744541525840759, 0.6827523112297058, 0.401719331741333, 0.18121743202209473, 0.9453489780426025, 1.2073357105255127, 1.4193720817565918, -0.26740363240242004, 1.5231125354766846, -0.1518346518278122, -0.631190299987793, 0.24915650486946106, 0.46395257115364075, -1.4391741752624512, 0.5438242554664612, 1.0724767446517944, 1.5701870918273926, -0.007758430205285549, 2.3413145542144775, -1.4843652248382568, 0.7330410480499268, 0.2456388920545578, 1.2121554613113403, 0.009496266022324562, -1.3779432773590088, 0.6469316482543945, -0.05609989911317825, 1.6031700372695923, 0.48122096061706543, -0.2896624505519867, -0.26427459716796875, 0.5525356531143188, 1.1997849941253662, 0.8276820778846741, -1.4407535791397095, -0.5697119832038879, -0.03366536274552345, -0.8725666999816895, 1.007677674293518, -0.44814586639404297, -1.4315499067306519, 0.5774762034416199, 1.0331817865371704, 0.7879738211631775, -1.390000343322754, 0.09627306461334229, 0.7275265455245972, -1.148184895515442, 0.572655439376831, 0.49069705605506897, -0.5430092811584473, 1.4274283647537231, -0.9991211295127869, -1.6859592199325562, 1.8428388833999634, -0.954918622970581, -1.0585365295410156, -0.6661391258239746, -0.7983158826828003, 0.5062333941459656, 0.16936689615249634, 0.4534885585308075, -1.0115692615509033, -0.18917959928512573, 0.1444670557975769, -0.0746852457523346, -0.9116672873497009, 0.46035197377204895, -0.3952273428440094, 1.0873850584030151, -0.26028212904930115, -0.2661001682281494, -1.507522463798523, 0.21940144896507263, -1.6335928440093994, 1.0229496955871582, 1.3891394138336182, 1.251083254814148, 0.7602461576461792, 0.2558537721633911, 1.324602723121643, 0.509345531463623, -0.6128595471382141, -0.005951999686658382, 0.5831069946289062, 0.14247916638851166, 0.45782986283302307, 1.0172820091247559, -1.4185831546783447, 1.4813549518585205, 0.10121937096118927, 0.23340760171413422, 2.1398496627807617, 1.5343942642211914, -1.8834725618362427, -1.2522051334381104, -1.5230220556259155, 0.17961664497852325, 1.096221923828125, -2.2345759868621826, 0.282528817653656, -0.4657563865184784, -0.4904399812221527, -0.11414813995361328, -1.543535828590393, 0.21531403064727783, 1.451464295387268, -0.9966568946838379, 0.3915158212184906, 0.5477311015129089, -1.7974838018417358, -0.9823133945465088, -1.1186598539352417, 0.609191358089447, 0.05222030356526375, -0.16674403846263885, 0.9604498147964478, 2.421884536743164, -0.8418182730674744, 1.6092865467071533, -0.34414803981781006, 0.4526604115962982, 1.7533276081085205, -0.15892525017261505, -0.5260391235351562, 0.9703062176704407, -0.2665533721446991, -0.909781277179718, 0.2096172273159027, -2.2711849212646484, 1.924042820930481, 0.5122870802879333, -1.2542411088943481, -0.013942182995378971, 1.5586785078048706, 0.687240719795227, -0.6839869022369385, 2.4821019172668457, -0.10129280388355255, 0.911770761013031, -0.19800613820552826, 1.0521488189697266, 0.3358182907104492, -1.3317396640777588, 0.20594654977321625, -1.1281044483184814, 0.6833444237709045, 0.4493505656719208, -0.5638540983200073, -0.01729341596364975, -1.2595616579055786, 0.53597491979599, -1.1061713695526123, 0.4549737572669983, 0.5105754733085632, 0.16186530888080597, -0.9170044660568237, 0.24867747724056244, -0.4190228283405304, -0.42905837297439575, -1.4513356685638428, 0.7918270826339722, 0.5565482378005981, -0.16231347620487213, 1.0281431674957275, -1.2162971496582031, 0.07351601868867874, 0.08537939935922623, -0.9594467282295227, 0.24127918481826782, -0.018363427370786667, -1.5747865438461304, -0.4299927055835724, -0.8635992407798767, 0.530124843120575, 0.04597553238272667, -1.2843464612960815, -0.6534234285354614, 0.027109257876873016, 0.21665118634700775, -2.3665122985839844, -0.8512499332427979, -0.4419780373573303, 1.1949368715286255, 0.01789780519902706, -0.9286752343177795, -0.4746997058391571, 0.977915346622467, 0.9349842667579651, 1.54069185256958, -1.536523461341858, 2.3455138206481934, -0.14198169112205505, -2.6189687252044678, 1.5994693040847778, -0.6375852227210999, 1.3447095155715942, 1.129291296005249, -0.31888341903686523, -1.1542701721191406, -0.5867912173271179, -0.19469743967056274, 0.347735196352005, 0.6587806344032288, -0.8458085060119629, -0.19581972062587738, -0.17463883757591248, -0.6335232257843018, 0.40984395146369934, 1.5696357488632202, 0.9104771614074707, 2.0652408599853516, -1.0768511295318604, -0.37985676527023315, -1.9500242471694946, 0.6775139570236206, 1.4962354898452759, -0.6666593551635742, 0.09079477190971375, 0.730454683303833, -0.8084608912467957, 1.1530344486236572, -0.5903280377388, -0.28540292382240295, 1.2223235368728638, 0.6935347318649292, 0.8795887231826782, 0.1682763695716858, 0.26725971698760986, 1.085720419883728, -0.04864250123500824, -0.46875908970832825, -2.3891310691833496, 1.1615854501724243, -0.7511511445045471, -0.2583385705947876, -2.2557358741760254, -0.0524425283074379, -0.7573074698448181, -0.3670361042022705, -0.38405367732048035, -0.9524039030075073, 1.6881093978881836, -1.6948726177215576, 0.6591914296150208, -0.2686830759048462, 0.5953869819641113, -0.1911715716123581, 0.5802314877510071, 0.9618479013442993, -0.5741850733757019, 0.6646643877029419, 0.46234044432640076, 0.13828164339065552, 0.5224442481994629, -1.3394834995269775, -0.7633345127105713, -1.18903648853302, 0.05496250092983246, -0.27802029252052307, 0.5786651968955994, -1.5452717542648315, -1.0648341178894043, 0.23352757096290588, -1.3029547929763794, 0.07857628911733627, 1.5730143785476685, 0.1637391746044159, 0.10824312269687653, -0.1277914196252823, -0.42999598383903503, -1.766538381576538, 0.7545820474624634, 0.6210219264030457, 0.7995128631591797, 0.8157569169998169, 0.6842353940010071, 0.5724793672561646, -0.20581349730491638, 1.1141761541366577, -1.3146401643753052, 2.246042251586914, 0.46493276953697205, -0.24895507097244263, 2.5487606525421143, -1.0608632564544678, 0.9755545258522034, 1.1293339729309082, -0.0690268874168396, -0.692689836025238, 1.1275438070297241, 0.731126606464386, 2.2694108486175537, -1.20839262008667, 1.268872857093811, -0.04111664742231369, -0.07870033383369446, 0.3965624272823334, 1.0124506950378418, 2.4979143142700195, 0.19615790247917175, -0.5145119428634644, 0.3628876209259033, 0.4622722864151001, 2.7363810539245605, -0.7998597621917725, -0.3456037938594818, 1.6066888570785522, -1.1105600595474243, -0.45141613483428955, -1.6033674478530884, 1.0115737915039062, 0.5551477670669556, 0.536274790763855, -1.3631665706634521, 1.2216331958770752, -0.012909522280097008, 0.9672514796257019, -0.07618781924247742, 0.35552605986595154, -0.487619012594223, -0.005352341569960117, 0.42450693249702454, 0.6437025666236877, 1.7371864318847656, 0.32369479537010193, -1.212384581565857, 0.8689746856689453, 0.06404700130224228, 1.90078866481781, 0.8159077167510986, -0.6841674447059631, 1.4843157529830933, 1.2445927858352661, 0.1765156239271164, 0.4878968894481659, 0.8121368885040283, -1.3101083040237427, 0.334838330745697, 0.5191136002540588, -0.26129773259162903, -0.38177749514579773, 0.23933464288711548, 1.4176520109176636, 1.0908260345458984, 1.085861086845398, -0.11073175072669983, 0.06620428711175919, 0.43313083052635193, 0.2435796558856964, 0.4346025586128235, -0.6368629932403564, 0.19101911783218384, -0.4841133952140808, -0.6931754946708679, 0.41030147671699524, -0.6905249953269958, -2.414255380630493, -1.6751619577407837, -0.5152009129524231, 0.34451577067375183, 0.6575925946235657, 0.034796856343746185, -0.09553594142198563, 0.2956981062889099, -0.6579058766365051, -1.7690595388412476, 0.722693681716919, -0.7417551875114441, -1.285127878189087, -0.11574596166610718, 2.770822048187256, 1.6643601655960083, -0.32142114639282227, -0.5391805768013, -0.6900668144226074, 2.1339645385742188, -0.3919580280780792, 0.2665964961051941, 0.8241143226623535, 1.0414892435073853, -0.6137703061103821, -0.2617092728614807, 0.15857747197151184, -1.4958173036575317, 0.45520374178886414, -1.7743446826934814, -0.213705912232399, 1.4213002920150757, -0.2534041702747345, -0.8561655282974243, -0.6440656185150146, 0.5904778242111206, 0.19628462195396423, 1.0158884525299072, -0.17174255847930908, -1.1059376001358032, -0.8792521953582764, -0.17019575834274292, 0.2319362908601761, -0.026582157239317894, -0.31033220887184143, -0.9416597485542297, -0.21651488542556763, -0.3338206708431244, 0.9204139113426208, -1.270147681236267, 0.8683702945709229, 0.8476831912994385, -0.23867905139923096, -0.6521851420402527, -1.3246893882751465, -0.034008778631687164, 0.05131406709551811, -0.3272147476673126, 0.3106590807437897, -2.926114320755005, -0.10744120180606842, -0.4361213147640228, 0.0012127063237130642, -0.36135509610176086, -1.7310503721237183, -1.6556893587112427, -1.4625730514526367, 1.1631031036376953, -0.7848412394523621, -0.005093366373330355, -0.34219393134117126, 0.21327900886535645, 0.329159677028656, 1.6431260108947754, 0.08173052966594696, 0.1489342600107193, 0.727754533290863, -2.561314105987549, -0.68802410364151, 0.6053786277770996, 0.199087992310524, 1.2019871473312378, 1.3230586051940918, -1.37620210647583, 0.45527079701423645, -2.1476516723632812, 0.3467504680156708, -0.5446518659591675, 1.3269274234771729, -0.13617557287216187, 1.0264688730239868, 0.40197280049324036, 1.360348105430603, -0.6972354054450989, -1.5460963249206543, -0.2991134226322174, -0.01599183864891529, -0.3709036409854889, -0.44026458263397217, 0.5695163607597351, 1.21592116355896, 0.23697564005851746, -0.61275315284729, -0.8891869187355042, -0.4979833960533142, -0.3131149113178253, 0.5247746706008911, 1.6626949310302734, 1.4779659509658813, 1.9135841131210327, 0.8353003263473511, -0.43028321862220764, 0.6003100275993347, 0.8568547368049622, -1.7150217294692993, 0.6744190454483032, -0.44205859303474426, 0.7100168466567993, -0.7205499410629272, -0.14628353714942932, 0.0690205916762352, -0.09771248698234558, 0.7066608667373657, 2.160653829574585, 0.8889755606651306, 1.6539007425308228, 0.12080756574869156, 0.12484470009803772, -0.2441089004278183, -1.235893964767456, 0.49265334010124207, -0.32966142892837524, -1.9871035814285278, -1.5728778839111328, -0.7277477979660034, 1.0598622560501099, -0.8423179388046265, 0.873518705368042, -0.599067747592926, -1.4372550249099731, 2.529390811920166, -0.30866414308547974, -2.0039167404174805, -0.27138400077819824, -1.498138427734375, 1.1771717071533203, 2.307400941848755, 1.171714425086975, -0.9349541664123535, -0.108412005007267, 0.7922626733779907, -1.0346109867095947, -0.36313101649284363, -0.10138928145170212, 0.13928306102752686, -0.5532732605934143, 0.6128029823303223, -2.16640043258667, 0.4558224380016327, 1.0110293626785278, -0.7322399020195007, -1.7108303308486938, -0.9870648384094238, -0.4612971544265747, -0.27781349420547485, -1.0228259563446045, 0.7991693019866943, -0.16952131688594818, 0.6528046131134033, -0.4747782051563263, -0.10984315723180771, 0.06956463307142258, -1.0107814073562622, -0.5724829435348511, -0.6616328358650208, 1.0120700597763062, -0.9882573485374451, 0.44233590364456177, -0.43428564071655273, 0.6038521528244019, 0.18947568535804749, -1.5619546175003052, -0.8489057421684265, 0.5569427013397217, 1.486868143081665, -0.1973685473203659, -1.4528917074203491, -1.347162127494812, 0.1332993507385254, 0.638177752494812, 0.8238223195075989, -0.3001087009906769, 0.4963351786136627, -0.7004372477531433, -0.9532080292701721, -1.6658446788787842, -1.7607721090316772, 0.10586593300104141, -0.5539195537567139, 0.30220475792884827, 0.19688273966312408, 0.4769406318664551, -0.814479410648346, -1.1226098537445068, -1.2429295778274536, -0.9215086698532104, 0.8259510397911072, -0.9427390098571777, -1.099910020828247, 0.6039740443229675, 2.0148491859436035, -0.23927012085914612, -0.07999066263437271, 0.27959686517715454, -2.3884332180023193, 1.297742486000061, -1.0944461822509766, 0.8123310208320618, -0.5111193656921387, 0.5587013363838196, -0.8155023455619812, 1.6884431838989258, -0.9935962557792664, 0.2537974715232849, -0.3475368320941925, -1.509006381034851, 0.9236048460006714, 0.056537311524152756, 1.3064428567886353, 0.29754868149757385, 1.4511491060256958, 0.7640834450721741, -0.4020419120788574, 0.6593146920204163, 0.5288928151130676, -3.013840675354004, -0.20190127193927765, -0.6037326455116272, -0.09279241412878036, -0.09413912892341614, 1.8321361541748047, -2.691108226776123, 1.6431398391723633, 0.5136600136756897, -0.8543826341629028, -0.6165029406547546, 0.4304846525192261, -1.677537441253662, 0.37650102376937866, 0.7588427662849426, -1.7152889966964722, 0.12553508579730988, 0.38040244579315186, -1.4969983100891113, -1.0688990354537964, 0.6800881028175354, -1.2935115098953247, 1.1413203477859497, 1.0726349353790283, -0.2702045440673828, 0.5141023993492126, -1.0356460809707642, -0.8285736441612244, -0.9469994902610779, 0.7985517978668213, 0.3679948151111603, -0.8340080380439758, -0.4770895838737488, 0.7328373193740845, 0.8798515796661377, -2.076493501663208, -0.9902433156967163, -0.31391099095344543, -1.55446457862854, 0.18900012969970703, -0.024274256080389023, 0.2981794774532318, 0.879306972026825, -0.2807719111442566, 1.5335983037948608, -0.4043358862400055, -1.0097242593765259, -0.5030971169471741, -0.6943662166595459, 0.10086271166801453, -0.49183493852615356, 2.197254180908203, -1.3659567832946777, -1.2981873750686646, 0.8173393607139587, -0.7133383750915527, -1.0143071413040161, 0.12458289414644241, -0.24624262750148773, -0.1505398154258728, -0.10209912061691284, 0.4121864438056946, 0.07976233214139938, 1.118911862373352, -0.03691506385803223, -0.23538759350776672, -0.9070298075675964, 0.5230510830879211, 0.020756902173161507, -1.7696698904037476, -0.25699710845947266, -1.87101411819458, 0.36398035287857056, -1.9445199966430664, 0.31446966528892517, 0.2507728040218353, -0.290083646774292, 0.22999730706214905, 1.3118070363998413, 0.5875757932662964, 1.0503816604614258, -0.41729050874710083, 0.4919750988483429, 1.5401874780654907, 1.0768057107925415, -0.3266197144985199, 0.0875009149312973, 0.2779151201248169, 0.5484063029289246, 0.6499075293540955, 0.6587702035903931, -1.1190714836120605, 0.11685089021921158, -0.018472392112016678, 2.0696396827697754, 0.14206562936306, -0.10733579844236374, 0.0027700664941221476, 0.9623634815216064, 0.8074020743370056, 1.599289059638977, 0.20035813748836517, -1.1443514823913574, -1.1242836713790894, -0.06508611142635345, 0.6961690783500671, -0.3491416275501251, 1.2772951126098633, 0.4682771563529968, -0.6906924247741699, 0.36352258920669556, -0.4187149405479431, -1.0984433889389038, 1.0184378623962402, -0.1622210592031479, 1.0441948175430298, -0.28657248616218567, -0.13163691759109497, 0.4202885627746582, -0.9091057777404785, -0.015288733877241611, 0.08266524225473404, 1.0825592279434204, -1.5234087705612183, 0.5352240204811096, -0.4284093379974365, 0.024544674903154373, 0.6464369893074036, -1.4768356084823608, -0.20120644569396973, -0.8704778552055359, 0.07619024813175201, -0.7206194400787354, 0.39677873253822327, 1.4083383083343506, 0.4835815727710724, -1.5809894800186157, -0.013008671812713146, 0.18633025884628296, 0.648575484752655, -1.290291666984558, 0.17854133248329163, -0.572229266166687, -0.5434862971305847, 0.3526652753353119, 1.3020198345184326, 0.7835859656333923, -0.14804820716381073, 0.22966153919696808, -1.178693413734436, -1.186322569847107, 0.29259148240089417, 1.3013328313827515, -0.9162296652793884, 0.2883927524089813, 0.25956422090530396, 2.116729974746704, 1.172530174255371, -0.9621651768684387, 0.46651986241340637, -0.5752492547035217, 0.3771471679210663, -1.3327723741531372, -0.487188458442688, 1.1923519372940063, 0.8180961608886719, 0.15618735551834106, 0.7010674476623535, 0.8785638213157654, -0.1671900898218155, 0.0787879154086113, 0.6774457097053528, 0.4767293632030487, -1.6549463272094727, 1.6657525300979614, -0.11394020915031433, 0.11870107054710388, 0.8570572137832642, -0.10427277535200119, 0.050301432609558105, -0.9340742230415344, 0.6104642748832703, 0.11762971431016922, -1.7952724695205688, 0.13489247858524323, -0.1224435567855835, -0.18897995352745056, -1.8601107597351074, 0.279095858335495, 0.98121577501297, 0.9433022141456604, -0.9130078554153442, 1.1789659261703491, -0.301825612783432, -0.07492481917142868, -0.10322485119104385, 1.8242392539978027, 0.1555992215871811, -1.7307581901550293, 0.1161176860332489, 0.342665433883667, -0.3324527442455292, -0.010036812163889408, -1.733599066734314, 0.12730619311332703, 0.530881404876709, 0.03483390063047409, -1.1459602117538452, -0.9488046169281006, -0.06655917316675186, -1.0710294246673584, 1.0499850511550903, 0.8694393038749695, 0.13885977864265442, 2.218641519546509, -0.01858259178698063, -0.9859170913696289, -1.3981472253799438, 0.1390564888715744, -1.4432716369628906, -0.5166954398155212, 0.7373476624488831, -1.912177324295044, 1.4934847354888916, -0.7084237337112427, -0.23329389095306396, -0.5856425762176514, 0.7589967846870422, 2.6107723712921143, 0.2585621178150177, -0.02673868089914322, -0.9616835117340088, 0.45784953236579895, -0.2996789515018463, 1.0583266019821167, -0.5823365449905396, -0.3995339870452881, -0.932155430316925, -1.0218608379364014, -0.8142666220664978, 0.35545089840888977, 0.05073096603155136, 0.26730307936668396, -0.4177590608596802, 1.0357155799865723, 0.7858202457427979, -0.2435615211725235, 0.5072228312492371, -1.1529130935668945, 1.171775221824646, 1.2477586269378662, -0.3167123794555664, -2.1232614517211914, -1.8208680152893066, -0.6614671349525452, -0.25044339895248413, -0.5019787549972534, 0.04410456120967865, -1.819079875946045, -1.2452878952026367, 1.8036115169525146, -0.4350431561470032, 0.11523275822401047, -1.2830212116241455, 0.08303294330835342, 0.8095195293426514, 2.957981824874878, -0.4282178580760956, 1.1238776445388794, 1.80380117893219, -1.2647814750671387, -0.4612412750720978, 0.41239604353904724, 0.7121344804763794, 0.2853018045425415, 0.2908762991428375, -0.4607365131378174, 2.5905375480651855, -1.3119416236877441, -0.4803539514541626, 0.42882901430130005, -1.227961540222168, -0.5214259028434753, 1.7669191360473633, -0.746910035610199, 0.7662705779075623, 0.7295708656311035, 1.0221147537231445, -0.014515545219182968, 0.5489262342453003, 1.2658113241195679, -0.12891441583633423, -1.1007411479949951, 1.610569953918457, 0.33000504970550537, -0.5087434649467468, -0.8083723783493042, -0.6035646200180054, -0.09738380461931229, -0.8681819438934326, 1.3209542036056519, -0.1771453619003296, -0.2608126699924469, 0.29000529646873474, -0.7537843585014343, 0.4632374346256256, -0.4365637004375458, 0.6035494208335876, -0.2149306833744049, 3.039027452468872, -1.6133639812469482, -1.1061567068099976, -0.7026221752166748, -0.6858844757080078, 1.9696227312088013, 1.461801528930664, -0.759321391582489, 0.6152717471122742, -1.1796423196792603, -0.6637836694717407, 1.1414387226104736, -0.08754312992095947, -0.29521626234054565, -0.17989559471607208, -0.983910083770752, 1.0404472351074219, -2.280069589614868, 0.31258901953697205, 0.7795277833938599, 1.1765973567962646, -0.7824281454086304, -1.0536106824874878, 0.2967253029346466, 1.2864067554473877, -1.5373127460479736, 0.6155847907066345, 0.4968651235103607, -1.0480186939239502, -1.2597185373306274, 0.08810848742723465, -0.01798367127776146, 0.98484867811203, 1.2539806365966797, -1.0679221153259277, -1.1069976091384888, -0.25828462839126587, -0.18453408777713776, -0.25373464822769165, -0.054529737681150436, -1.5295861959457397, -0.057242847979068756, 1.1312012672424316, 0.19332265853881836, -0.17844952642917633, 0.735837459564209, 1.096148133277893, 1.044312596321106, -0.3337402045726776, 0.7246315479278564, -0.6252380609512329, 0.470927894115448, -0.5628266930580139, 1.592727541923523, 0.3129967451095581, -0.8447611331939697, -0.9561479091644287], [0.3032020628452301, -0.2812279164791107, -0.3286941945552826, 1.3763463497161865, 0.7047802805900574, 0.49605947732925415, 0.3123827278614044, 0.24979953467845917, 0.07992047816514969, -0.26882442831993103, 1.1594685316085815, -1.6735846996307373, 0.6092213988304138, 0.8791573643684387, 1.5126969814300537, 1.6446834802627563, 1.2782191038131714, 1.0451759099960327, -0.8284094929695129, -1.4861027002334595, 1.1608457565307617, 0.5969526171684265, 1.326509714126587, 0.08673001080751419, 0.6020064353942871, -0.4118880331516266, -0.18610528111457825, 0.5081002712249756, 0.4488624930381775, 1.2793464660644531, -1.170494556427002, -0.05964558571577072, -1.007952094078064, -0.8079850077629089, -2.2338480949401855, -0.30280593037605286, -0.24104395508766174, 0.9908524751663208, -1.1805092096328735, -1.242125153541565, 0.7012291550636292, -0.1442052274942398, -0.15895000100135803, 2.2971227169036865, -0.3425185978412628, 1.335663914680481, 1.179071068763733, 0.7591623067855835, 0.6118133664131165, 0.7405585050582886, 0.3155306577682495, 1.0780091285705566, -0.7666285634040833, -1.0826919078826904, 0.9991742968559265, 1.0319445133209229, -1.9011834859848022, -0.09743797034025192, 0.3812086284160614, -0.0777047723531723, 0.3001578748226166, 0.6697458028793335, -1.1975644826889038, -0.5520467758178711, -0.13416774570941925, -0.6416478157043457, 0.8567820191383362, -0.7126350998878479, 1.235770344734192, -0.06387679278850555, -0.4662803113460541, 0.760368287563324, 1.5334863662719727, 0.5248400568962097, 0.2440868765115738, 0.3569759726524353, 0.7730039954185486, 0.4536797106266022, 0.8341549634933472, 0.3410887122154236, 1.9913091659545898, -0.05836431309580803, -1.5930712223052979, 0.05102551728487015, -0.8558953404426575, -0.2929432690143585, 0.36000868678092957, -0.3149910569190979, -0.16397273540496826, 0.5155863165855408, 0.971656322479248, -2.660945177078247, -0.8126106262207031, -0.2293277531862259, -1.0982091426849365, -0.09121165424585342, 0.9527993202209473, -0.7672858238220215, 0.5259333252906799, 0.4430236518383026, -0.8554709553718567, 2.1054527759552, 0.8555246591567993, 0.5842021107673645, 0.48595693707466125, 1.0105557441711426, 1.2946501970291138, -0.6163619756698608, 0.8248018026351929, -0.45416492223739624, 0.5749725699424744, -0.1022026315331459, 2.158607244491577, -0.3072648346424103, -3.2590115070343018, -0.7974748611450195, 1.129050612449646, -0.387942910194397, -0.9868456721305847, 1.5785183906555176, -0.3511180281639099, 0.9758161902427673, -0.29131439328193665, -0.039982717484235764, -0.21294276416301727, -1.1360350847244263, 0.088084377348423, -1.388845682144165, 1.0150146484375, -0.6037147045135498, -0.6359200477600098, -1.8266412019729614, 0.19276075065135956, -0.10899341106414795, 0.046639543026685715, -0.00784299336373806, 0.5535886287689209, 0.06475462764501572, -0.2969524562358856, 0.732367992401123, 0.02744915336370468, -2.007467746734619, -0.4369421899318695, -0.7527724504470825, 0.1082964539527893, 0.29027196764945984, 0.6185666918754578, -1.0762513875961304, 1.6546145677566528, 1.3937511444091797, -0.08105926215648651, 0.7192439436912537, 1.2137938737869263, -0.4619804322719574, -0.5959659218788147, 2.0659046173095703, 0.010039339773356915, 1.3001796007156372, -0.14695408940315247, 1.3284815549850464, -0.6259031891822815, 1.9290616512298584, 1.338486671447754, 0.20763637125492096, 0.556964099407196, 0.43470779061317444, -0.6088477969169617, 1.2865347862243652, -0.03720925748348236, -0.5835896730422974, -0.441180944442749, -0.2721324563026428, -0.09853555262088776, 1.09935462474823, -1.3031765222549438, 1.3759357929229736, 1.439986228942871, -1.7302055358886719, 0.20497195422649384, -0.5466497540473938, 0.5542654395103455, 0.9940707087516785, -0.5146107077598572, 1.1942182779312134, 1.1473495960235596, -0.3976012170314789, 0.7156803607940674, -0.4771276116371155, 1.011240839958191, -1.0956571102142334, 0.4200785160064697, -0.718208909034729, -0.5666312575340271, 1.0329654216766357, -1.936264991760254, -0.04373999685049057, -1.7880240678787231, 0.9652934074401855, -0.905404806137085, -2.086630344390869, -0.40750572085380554, -0.6725688576698303, -0.6148831844329834, 0.7385624647140503, 0.9840591549873352, -0.6616753935813904, 0.3647979497909546, -0.6340709924697876, -2.0298683643341064, 0.8585900664329529, 0.9605386853218079, -0.33456000685691833, -0.44370484352111816, -0.34301790595054626, -0.9952773451805115, 1.9226304292678833, -1.2341853380203247, 1.165136694908142, 0.16648808121681213, 0.8564146757125854, 0.03771034628152847, 0.44658395648002625, -0.940000593662262, -0.20088012516498566, -0.9657501578330994, 1.218039870262146, 1.7221945524215698, 0.9510058164596558, 0.2410186231136322, 1.574759840965271, 0.4937630593776703, -0.34735026955604553, -0.5092150568962097, -0.2090061902999878, 0.28141188621520996, -1.6628435850143433, 1.3008843660354614, 1.1147873401641846, -1.3741662502288818, -0.6325594186782837, 0.7029621005058289, 1.7964783906936646, 0.6858892440795898, 0.35627374053001404, -0.6411710977554321, -0.4580991268157959, -1.1050018072128296, -0.32243308424949646, -0.027582265436649323, 1.5715339183807373, 1.0218244791030884, -1.3848687410354614, -0.450813889503479, -0.5184793472290039, 1.2258214950561523, 0.5358767509460449, -0.13100264966487885, -0.46015870571136475, -1.1767737865447998, -0.5278720855712891, 2.0432705879211426, 0.2517964541912079, -0.3136647939682007, -0.5056495070457458, 0.009736672043800354, -0.7104995250701904, -1.9905834197998047, -1.9906736612319946, 0.1684105545282364, -1.7568470239639282, -1.4566909074783325, 0.21912428736686707, -1.1057742834091187, 0.8789428472518921, 1.3142509460449219, -0.3661544919013977, 1.569286823272705, -1.9506897926330566, -0.8664640188217163, 0.1931246817111969, 1.2671966552734375, -0.7110217809677124, -0.5279020071029663, -0.6664246320724487, 1.9466919898986816, -0.5551761388778687, 1.673733115196228, -0.9646006226539612, 1.0786621570587158, 1.197455644607544, -0.6067059636116028, 0.060592565685510635, 0.3669642508029938, 1.0989251136779785, -0.15097539126873016, 1.0414576530456543, 0.6504845023155212, -1.3327114582061768, 1.6897990703582764, 1.1690099239349365, 0.8634048700332642, 0.9085938930511475, 1.1039994955062866, -0.9932443499565125, -0.18234899640083313, 0.5897743105888367, 1.3365302085876465, -1.6068730354309082, 1.896933674812317, -0.2203855961561203, -0.3480116128921509, -0.7814975380897522, 0.38170164823532104, -0.07243519276380539, -0.7842725515365601, 0.11009299010038376, 1.0455267429351807, 1.1291210651397705, -0.09668724238872528, -0.2244657278060913, 0.8401826024055481, -0.38644638657569885, 1.7882100343704224, -1.7471833229064941, -0.1981218159198761, -1.1099698543548584, 0.972819447517395, 0.9519034624099731, 1.3347718715667725, 0.8231246471405029, -0.6626474261283875, -0.511655330657959, -1.4990614652633667, -0.04189189523458481, 1.1948909759521484, -1.7309921979904175, 0.3167477548122406, 0.47091957926750183, 1.814752221107483, 0.8455201983451843, -1.433896541595459, -0.05098698288202286, 0.5207961201667786, 0.37601783871650696, -0.7188447117805481, -0.730621337890625, 0.4258780777454376, -1.4776967763900757, -0.27483102679252625, 0.8211326599121094, -0.12576812505722046, -1.1629278659820557, -2.707618474960327, -0.4867088794708252, 0.5643221735954285, -1.1530901193618774, 2.5460562705993652, -1.204866647720337, 0.0700300931930542, -0.44406595826148987, 0.30147162079811096, -1.5515512228012085, 0.031251031905412674, -0.6618654131889343, -0.6966652274131775, 0.10054781287908554, -0.7371176481246948, 0.7269505262374878, -1.1731735467910767, 0.7082601189613342, -0.9707944393157959, 1.0043644905090332, 0.1738317608833313, -0.9776117205619812, 0.869177520275116, 2.27310848236084, 1.8630422353744507, 0.19640395045280457, 1.233793020248413, -1.4681804180145264, 0.509888231754303, 0.8120899200439453, 1.270747423171997, -0.603148341178894, -1.6970539093017578, 3.0174803733825684, -1.0007610321044922, -1.8466269969940186, 0.9760461449623108, -0.35584911704063416, -0.44069769978523254, -0.8756442070007324, -1.4644527435302734, -0.5057465434074402, 0.49680301547050476, 0.765189528465271, -1.0251346826553345, -0.37766706943511963, -0.3775383234024048, 1.2797740697860718, -0.3472224175930023, 1.3865917921066284, 1.9276820421218872, 0.9216165542602539, 0.27125075459480286, 0.6810253858566284, 0.35351958870887756, -0.8533799052238464, 0.475273072719574, -1.3975175619125366, 0.24445325136184692, 0.07563644647598267, 0.1552857756614685, -1.6857073307037354, -0.37774497270584106, 0.0971526950597763, -0.7602825164794922, -0.2866324484348297, 1.1666909456253052, -0.4456925392150879, -1.3860924243927002, 0.029064299538731575, -0.5983414649963379, 0.9502568244934082, 2.139598846435547, 1.1894487142562866, -0.40748271346092224, 1.2166694402694702, 2.325007438659668, -1.218770980834961, 0.5628741383552551, 0.6127984523773193, 1.7641547918319702, 1.3835327625274658, 0.26941561698913574, -0.027253491804003716, 0.6490781903266907, -0.461237370967865, 0.09364566206932068, 0.762426495552063, -1.1148128509521484, -1.6530065536499023, -1.0404309034347534, -1.2853659391403198, -2.6050262451171875, -0.6758223176002502, 1.111555576324463, -0.9092049598693848, 0.8264933824539185, -1.1509976387023926, -0.15253038704395294, -2.096129894256592, 1.9376499652862549, -1.0775282382965088, 1.210486650466919, -0.6722957491874695, 0.34896227717399597, 1.1516693830490112, 0.337877482175827, 0.056203845888376236, -0.6906777620315552, 1.3510057926177979, 2.108232259750366, -1.1473920345306396, -2.032897710800171, -3.0315730571746826, 1.728684425354004, -0.4675760567188263, -0.9760613441467285, -1.4936578273773193, 1.1714203357696533, 1.2840960025787354, 0.17748402059078217, 0.39681825041770935, -1.2069404125213623, -0.6678543090820312, 0.8210066556930542, 0.7764281630516052, -0.8453957438468933, -2.0097928047180176, 0.2281973958015442, 1.5451371669769287, -0.060594093054533005, 0.320761501789093, 0.058648452162742615, 1.4309803247451782, 0.53016597032547, 0.07283097505569458, -0.7989058494567871, -0.7087357044219971, -1.2323356866836548, -1.0727996826171875, -0.24653410911560059, 0.5659011602401733, -0.14294326305389404, 0.6116780638694763, 1.368011474609375, -0.5387786626815796, 0.6283192038536072, -2.77097225189209, 0.13291539251804352, -0.6038822531700134, 1.1197752952575684, -1.6094825267791748, 1.4097669124603271, -1.8708182573318481, -0.24797482788562775, -0.07120781391859055, -0.697303295135498, 0.3952004313468933, 1.6134121417999268, -0.8365116715431213, 0.6124264001846313, 0.4320645034313202, 0.6059222221374512, 1.4209119081497192, -0.04057065397500992, -2.1938817501068115, 1.010258674621582, 0.05676195025444031, 0.5760098695755005, 0.07710369676351547, -0.3181638717651367, 0.19957242906093597, 0.41447222232818604, 0.4820304811000824, 0.8933266997337341, 0.4510977864265442, 0.07309114187955856, 1.1385016441345215, 0.8178542256355286, -0.6033463478088379, -1.0870921611785889, -0.4148581624031067, 0.7940049767494202, 0.14694705605506897, 0.2759969234466553, -1.0685453414916992, -2.289163112640381, -1.0178662538528442, -0.7609929442405701, -2.7154746055603027, 0.48387256264686584, 1.249771237373352, -0.1657020002603531, -0.009041334502398968, 0.6079645752906799, -0.17773310840129852, 0.7790343761444092, 0.44226300716400146, 2.0906996726989746, 0.24740827083587646, -1.5406185388565063, -1.4947863817214966, 2.0703914165496826, -1.0281447172164917, 1.7893469333648682, 2.257458448410034, 0.15675130486488342, -0.37213680148124695, -1.4453545808792114, 0.3697666525840759, 0.24787913262844086, -2.2464122772216797, 0.468107670545578, -0.19724953174591064, -0.27476635575294495, -0.49181070923805237, -0.2914729118347168, -0.9558655023574829, 2.0228700637817383, 1.8930728435516357, 0.3009120225906372, -0.4308438301086426, -0.7441383600234985, -1.8061511516571045, 0.8385403752326965, 1.3929728269577026, 0.899080216884613, 0.3772350251674652, 0.2675115168094635, 0.4262528419494629, 0.6250090599060059, 1.2971123456954956, 0.7884466648101807, 0.17325475811958313, -0.38770583271980286, -1.4500102996826172, 1.082809329032898, 0.46834322810173035, -1.5293388366699219, 0.5730686783790588, 0.12976059317588806, -0.18013668060302734, 0.9244193434715271, -1.809429407119751, 0.34276846051216125, -0.6890989542007446, 0.7219242453575134, -0.49706587195396423, -0.8038952946662903, 1.9852544069290161, -1.3106316328048706, 0.9325685501098633, -1.5854668617248535, 0.3259095251560211, -0.3622279167175293, 0.5151849985122681, 0.965059220790863, 1.4463292360305786, 0.4798477590084076, 0.6920856833457947, -0.5799676775932312, -0.550460934638977, -1.025757908821106, -0.07100382447242737, -1.127359390258789, 0.5589994788169861, 0.7580443620681763, 0.15222829580307007, 0.4250496029853821, 1.4881302118301392, 0.4274671673774719, -0.6643391847610474, -1.6715974807739258, 1.046225666999817, 0.25696930289268494, 0.5821143984794617, -1.165071725845337, -1.3747913837432861, -0.37389326095581055, 0.08076027780771255, -1.32880437374115, 0.7171902060508728, -0.4249682128429413, -0.9670329689979553, 0.2750348746776581, -1.2819139957427979, 0.03948147967457771, -0.627113401889801, 0.9881969094276428, -1.551822304725647, 1.119197130203247, -0.7371939420700073, -0.8110275864601135, -1.3527923822402954, 0.020299652591347694, -0.14290353655815125, 0.8032167553901672, 0.0037268390879034996, 1.6146240234375, -1.2171584367752075, -1.3273799419403076, -1.4981107711791992, 0.288751482963562, 1.7486125230789185, 0.7972779870033264, -0.07863350212574005, 0.8874497413635254, -0.6245944499969482, 1.066727876663208, 0.7356656789779663, -0.15470941364765167, 0.9238021373748779, 0.7296295166015625, 0.7759871482849121, -0.9319853186607361, -0.2752368152141571, -0.30757999420166016, -0.08465855568647385, 1.018722653388977, -0.2431487739086151, -1.1301593780517578, 0.6783754825592041, 0.9380000829696655, 0.5650469660758972, -0.09020300209522247, -0.14624491333961487, 0.9027988910675049, -0.04272949695587158, -1.0138860940933228, 1.0004103183746338, 0.5164874196052551, -0.16725417971611023, 0.29917436838150024, 0.3092859387397766, -0.7044621706008911, -0.24501633644104004, -0.7120646238327026, -0.11386289447546005, 1.9212994575500488, 0.1401003748178482, 1.4460079669952393, 0.5050613880157471, 0.16174538433551788, -0.19682498276233673, 0.9758164286613464, -0.32855159044265747, -0.6578095555305481, -2.083660125732422, -0.8141393661499023, -0.9721030592918396, -0.18962597846984863, 0.4346785843372345, -1.6797062158584595, -0.40153419971466064, -0.707933247089386, -0.7256770730018616, 1.5313705205917358, 0.30713632702827454, 0.016562478616833687, 0.5878003835678101, 1.4256144762039185, 0.5356202721595764, -0.547885537147522, 0.2325562834739685, -0.9410383701324463, -0.5736693739891052, -0.9024368524551392, 0.7237028479576111, -0.027855103835463524, 1.2519710063934326, 0.03733647242188454, -1.2933943271636963, -0.6971082091331482, -0.1195596382021904, -1.2851625680923462, 0.7804551124572754, 1.408461093902588, -1.7025070190429688, -3.4321062564849854, -0.40992271900177, -0.3335559368133545, 1.46726655960083, -0.6254160404205322, -0.574678897857666, 0.7675760388374329, 1.0262038707733154, 0.14070917665958405, 1.3324973583221436, 1.5344561338424683, -0.7393686175346375, -0.3605683743953705, 0.315106064081192, 1.6722828149795532, -0.6173315048217773, -1.0779739618301392, -0.9761984348297119, -1.1383488178253174, 2.034043312072754, 0.6102861166000366, -0.7789098620414734, 0.9268432259559631, 1.64338219165802, -0.8523643612861633, 0.4900572896003723, -0.9248837828636169, 1.0691941976547241, 1.2694436311721802, -1.315871000289917, 0.07319042086601257, -0.18187937140464783, 1.2603285312652588, -0.6428700089454651, 0.10368449985980988, -0.49117955565452576, 0.6107009649276733, -0.31736084818840027, -0.8185932636260986, -0.4176580607891083, 0.760546863079071, -0.19967225193977356, 0.8331854939460754, -0.21435676515102386, 0.8742045760154724, -0.2568749785423279, 0.019785011187195778, 0.3045486509799957, 0.8286916017532349, -1.4023250341415405, -0.32487916946411133, -0.41857001185417175, -2.2462856769561768, 1.8151930570602417, 0.09221091866493225, 1.203925609588623, -0.4730379283428192, -0.05600171908736229, -0.7599607110023499, 0.8601998090744019, -0.07281488180160522, 0.6146404147148132, -1.1969631910324097, 0.40837982296943665, 0.9584253430366516, 0.20041784644126892, 1.3469454050064087, -0.2597811222076416, -0.8866016268730164, -0.05555028095841408, -1.6144578456878662, 0.93165522813797, 1.239410161972046, 0.9767245650291443, 0.325610488653183, -0.782732367515564, 0.04304676130414009, -1.8684269189834595, -1.8383761644363403, -0.15339039266109467, 0.06928346306085587, 0.5342609882354736, -1.1544599533081055, -0.8448238968849182, 0.4491417706012726, 0.195865198969841, 2.2135133743286133, -0.7623093724250793, -0.4460347890853882, -0.31358519196510315, -0.5150779485702515, 1.2117376327514648, 0.24279068410396576, 1.0096951723098755, 1.5824013948440552, 0.4063451290130615, -0.05654120817780495, 1.098880648612976, -0.6052179932594299, -2.766136407852173, -0.09194057434797287, 0.07950567454099655, 0.4703625440597534, -0.6055943965911865, -1.0235577821731567, 0.5768510699272156, -1.4573432207107544, 0.16396324336528778, -0.20961956679821014, -0.4241710901260376, -0.9955577850341797, 0.00856100581586361, 0.8085759282112122, 2.397782802581787, 1.9003978967666626, 0.1738041490316391, 0.5172938704490662, 1.431364893913269, 0.7983782887458801, -0.4208776652812958, 1.006115198135376, -0.18688856065273285, 1.5058610439300537, -0.7914252877235413, 1.149126410484314, 0.004632963333278894, -0.4239986836910248, -0.4283052086830139, 0.6314089894294739, 1.2821695804595947, -0.3951476514339447, 0.2834312319755554, -0.4120129644870758, 0.09399979561567307, -1.9659202098846436, -0.40435317158699036, -0.9236272573471069, 0.07832646369934082, -0.27718254923820496, 0.7333930134773254, 0.8893153667449951, -0.06345344334840775, -1.3033363819122314, 0.30525878071784973, -1.779096007347107, 0.4499358832836151, 0.6497489809989929, -0.2923675775527954, 0.7388530969619751, -0.25862956047058105, 0.04044435918331146, 0.41546374559402466, 0.5867606401443481, -0.7563100457191467, -0.8054404854774475, 0.9880244135856628, 1.0137375593185425, -0.6451442241668701, 1.0080673694610596, -0.30792859196662903, -0.4801948070526123, -1.1419121026992798, -1.8304564952850342, 0.1993722766637802, -1.130347490310669, 0.08489292860031128, 0.2623484134674072, 0.4166485369205475, 0.22143633663654327, -0.599442183971405, 0.3084028959274292, -0.06719134747982025, 0.243153914809227, -0.30702030658721924, 0.061610929667949677, 0.7646277546882629, -0.8254117369651794, -0.028895579278469086, -0.2666237950325012, -0.6335923671722412, 0.12096454203128815, 0.1342163383960724, 0.8787917494773865, 1.0054738521575928, 0.85160231590271, 2.6963655948638916, -0.19874216616153717, -0.050848588347435, 0.523630678653717, 0.1672799289226532, 1.9407603740692139, -1.1155786514282227, -0.2745507061481476, -0.6225749254226685, -1.457850456237793, -0.1929175853729248, -0.6617060899734497, 0.7663554549217224, -0.6038591861724854, 0.24852152168750763, 0.08298346400260925, 1.5763676166534424, 1.6391313076019287, -1.4481747150421143, 0.5971355438232422, -0.16592612862586975, 1.0624284744262695, 1.1598234176635742, -0.7203028202056885, 0.5838346481323242, 0.3161994218826294, -0.14695939421653748, 1.014790654182434, 1.1028324365615845, 0.7729520797729492, 1.7390327453613281, -0.5400843620300293, -0.5192572474479675, 1.2845882177352905, -0.35541751980781555, 1.4400684833526611, 2.180295944213867, 0.26063260436058044, -0.19716016948223114, -1.4475758075714111, -2.1465096473693848, 0.8403908610343933, -0.15831464529037476, 0.6888819932937622, -1.4096928834915161, -1.7946618795394897, 0.12051233649253845, -0.8238824009895325, 0.14467090368270874, 1.3004608154296875, -0.18198971450328827, 0.5477892756462097, 0.047016121447086334, -1.6413958072662354, -0.21410688757896423, -0.7474639415740967, 0.5535873174667358, -0.49623996019363403, 0.8828937411308289, -0.061964549124240875, -1.196812391281128, -1.5537490844726562, -1.634748935699463, -1.1122550964355469, -0.8788250088691711, 1.625690221786499, 1.0931156873703003, -0.3646259903907776, -0.25362053513526917, 0.1804065853357315, 1.5245145559310913, -0.2366802990436554, -0.33078426122665405, -1.1459234952926636, -0.5886718034744263, -0.9700354337692261, 0.4549710154533386, -2.1995861530303955, 1.6526057720184326, 1.6265634298324585, 0.5289729237556458, -1.6327348947525024, -0.07478423416614532, 0.8603187799453735, -0.031691256910562515, 1.0652427673339844, 1.2970784902572632, -0.0865042433142662, -1.207722783088684, -0.55190509557724, 0.6302798390388489, -0.6155499219894409, -0.4618386924266815, 1.0948625802993774, 1.1585588455200195, -0.35700809955596924, -0.4441000819206238, -1.669751763343811, 1.6543782949447632, 2.9232122898101807, 0.46891501545906067, -0.27784547209739685, 0.7299525737762451, 0.5110089778900146, -1.3530350923538208, 0.7786989212036133, 0.7693207859992981, -2.212308168411255, -1.6497851610183716], [1.2649011611938477, -0.09242226183414459, -1.9051365852355957, -1.090076208114624, -0.1402711123228073, -0.46266523003578186, 2.261575222015381, 0.807632565498352, -0.6521930694580078, -2.3253679275512695, -1.4895099401474, 1.6368516683578491, 1.022820234298706, -2.0297675132751465, 0.24714435636997223, 0.20645467936992645, -0.35332056879997253, 1.0738955736160278, -1.2754873037338257, 1.958993911743164, 0.3313779830932617, -0.4559965431690216, 0.9388493299484253, 0.057113248854875565, -0.3082237243652344, -0.5406613945960999, 0.018178213387727737, 0.3295730650424957, 0.743679404258728, 0.8466780185699463, 0.7338036894798279, 0.7706102132797241, 0.31762802600860596, 0.8620390295982361, 1.727563500404358, -0.5183319449424744, 0.163762629032135, 0.48308610916137695, 0.466998428106308, -0.638698399066925, -0.8010485172271729, -1.1158608198165894, 0.3138194978237152, -2.0336785316467285, 0.06173110380768776, -1.0155316591262817, -0.7796497941017151, 0.22530566155910492, 0.41815826296806335, -0.4998819828033447, -0.1838218718767166, -0.9709266424179077, 1.3231850862503052, -1.7626855373382568, 0.28436553478240967, 0.25366833806037903, -1.734471082687378, 0.8106217980384827, 1.189095377922058, -0.8374720811843872, -0.007779983803629875, 0.22859448194503784, 1.3795970678329468, 0.04330289363861084, 0.5512793660163879, -0.1694137454032898, 0.42759302258491516, 0.4705815315246582, -0.6831601858139038, -1.1162569522857666, 1.7484130859375, 0.6632391810417175, 0.6729125380516052, -0.16181577742099762, -1.0259628295898438, 1.3004491329193115, 2.5121350288391113, -1.0377833843231201, 0.6359668970108032, 0.44815489649772644, -1.401646614074707, -1.2703231573104858, -0.6179333925247192, -0.9798791408538818, 0.4711403250694275, -0.4784310460090637, 0.5710858106613159, 0.25614601373672485, 0.14207473397254944, -0.6499490737915039, -1.8110064268112183, -1.1062620878219604, -0.8902794718742371, -0.15846958756446838, 0.16482599079608917, 0.2681290805339813, 1.133911371231079, -1.3847836256027222, -1.5272654294967651, 0.3255191743373871, 1.8488470315933228, 0.7338463068008423, -1.816481113433838, -0.148462176322937, -0.4036194980144501, -0.3511331081390381, -1.5743157863616943, -0.5405141711235046, 0.5399962067604065, -1.3382655382156372, -0.9067837595939636, 0.8953543901443481, 1.5557957887649536, -0.7352631688117981, 0.37591099739074707, -1.7724826335906982, -0.6375856399536133, 0.3874341547489166, 1.4885700941085815, 0.18975558876991272, 0.6334609985351562, -0.03715799003839493, 0.8871155977249146, 0.9063979387283325, -0.43114185333251953, 2.8824691772460938, 1.22135329246521, -0.9648966789245605, -1.7319509983062744, -0.6488091945648193, 0.4037320911884308, -1.4188531637191772, -1.4043536186218262, 1.4303902387619019, 0.3640689551830292, -0.22540314495563507, -0.3027404844760895, -0.7288388609886169, 0.5716012716293335, -0.010900617577135563, -0.250169575214386, -0.14765813946723938, -1.6755841970443726, 0.8882708549499512, -0.08133488893508911, -1.3172268867492676, -0.1078491061925888, -0.22825296223163605, -1.595962643623352, -1.218091607093811, -1.3600738048553467, 0.1094757467508316, 1.2006779909133911, -0.8539357781410217, 1.1941916942596436, 0.04823216050863266, -0.037423618137836456, 0.676125168800354, -0.8357859253883362, -0.07287885993719101, 2.0918080806732178, 0.543605625629425, -0.4444386661052704, 1.9346221685409546, 0.9369210004806519, -0.9090678095817566, 0.7620170712471008, 0.4396475851535797, -0.23262734711170197, 2.368741512298584, 1.5309454202651978, 1.7159587144851685, 0.8913130164146423, 0.4485720694065094, -0.91712486743927, 0.18932317197322845, 1.7113115787506104, -0.4912983477115631, -1.0684393644332886, -1.1480841636657715, -0.9960951805114746, -0.035770367830991745, 0.29578688740730286, -1.093334436416626, -0.9430482983589172, 0.7479270696640015, -1.1220420598983765, -0.15830405056476593, -0.9331826567649841, -1.202030062675476, 0.7381606698036194, -0.5692939162254333, -0.40699872374534607, 0.4229249060153961, -1.9076292514801025, 0.16420896351337433, 0.7826024889945984, -0.07382804155349731, -1.1226929426193237, -1.3723902702331543, 0.19301167130470276, -1.9420201778411865, -1.5934643745422363, 0.6359946727752686, -0.43192675709724426, 0.788406252861023, -0.3660813868045807, 0.7491406798362732, -0.29054486751556396, -0.36394673585891724, 1.3044041395187378, 0.4696347415447235, -0.5355244874954224, 0.011673176661133766, 1.1983919143676758, -0.6204213500022888, 0.5830898880958557, -0.6833877563476562, 0.7360320687294006, 0.4918457865715027, -1.0913124084472656, -1.4496251344680786, 0.4072376787662506, 0.05454033613204956, -0.27605563402175903, -0.7305122017860413, 0.959043025970459, 0.6570728421211243, 1.8542262315750122, 0.09724099189043045, -2.4453773498535156, 1.682188868522644, -0.14620669186115265, 0.5909804105758667, -0.41997000575065613, 1.1402047872543335, 0.43670642375946045, 1.2479203939437866, -0.4934293031692505, 1.618859052658081, 0.9500679969787598, -0.65043044090271, -1.1021982431411743, -1.6052491664886475, 1.3255345821380615, -0.25877895951271057, -0.5126309990882874, -1.6078189611434937, 0.1660364270210266, -0.9338616728782654, 0.16942214965820312, -0.4424457252025604, 0.3159830868244171, 0.11254176497459412, 1.2204915285110474, 0.4838796854019165, 0.33874714374542236, 0.726130485534668, -1.4202229976654053, 0.41469013690948486, -0.9597393274307251, -0.29857975244522095, -0.33559250831604004, 1.1367484331130981, 0.23898163437843323, -0.5946319699287415, -0.6881393194198608, -0.7216969728469849, 2.006784439086914, -1.0081127882003784, -0.5473829507827759, 1.1626838445663452, 1.8044283390045166, 0.21011927723884583, -0.13745509088039398, 1.7559677362442017, 0.38588303327560425, -0.16916070878505707, 1.1025574207305908, -0.09939206391572952, 0.18244817852973938, 2.267425060272217, -2.042121648788452, -0.07591046392917633, 2.084256410598755, -1.0181854963302612, -0.7630444169044495, -0.03530976548790932, 0.8487787842750549, -0.5728062391281128, -0.2428373247385025, -0.7512421607971191, 0.8570725321769714, -0.6427517533302307, 1.1913739442825317, 0.820043683052063, -2.3884034156799316, 0.6668171286582947, 0.3439737558364868, 0.9684372544288635, -0.07126174122095108, 1.3821109533309937, -1.1869629621505737, 0.3438032567501068, 0.9098796844482422, -0.8038597106933594, -0.9751349687576294, 1.423883080482483, -1.4089252948760986, 0.40518316626548767, -0.3258666396141052, -0.4200855791568756, -0.7178781032562256, 0.2122015655040741, 1.6965065002441406, 0.7210995554924011, 0.2665286958217621, 0.0020248747896403074, 1.5293084383010864, -0.12044025212526321, 0.04419434443116188, 0.1472318321466446, 0.4772709608078003, 0.1785438060760498, 0.530292272567749, 1.2306382656097412, 0.2719820737838745, -0.22643327713012695, 0.5438506007194519, 0.5614018440246582, -0.4394558370113373, 0.05249054357409477, 0.5925112962722778, -0.3773426413536072, 0.6681374311447144, -0.17148081958293915, 0.1592274308204651, -0.6789504885673523, -0.44902363419532776, -1.2357391119003296, -1.0776653289794922, -0.2556191086769104, -0.5046079754829407, 0.02116202749311924, -0.16935472190380096, 1.2555465698242188, -0.0024763380642980337, -0.05555817857384682, 0.3184332549571991, 0.5097029209136963, 0.3899315893650055, -0.7526304721832275, 0.4691905379295349, 0.49949607253074646, 0.12962262332439423, 0.24078495800495148, 0.034393131732940674, -1.9515836238861084, 1.4711291790008545, -0.5405497550964355, -0.08939753472805023, 0.2231249064207077, 1.1738355159759521, -1.2656229734420776, 0.962539792060852, -0.46435582637786865, 0.7389695048332214, 0.608353316783905, 1.005480170249939, 1.1960712671279907, -0.5330806374549866, -0.16991187632083893, -1.1963979005813599, 1.0651811361312866, 1.0758681297302246, -1.1770414113998413, 0.47289180755615234, -0.7921066880226135, -0.44927358627319336, 0.4279690086841583, -1.2320407629013062, -0.2802134156227112, 0.36496296525001526, -0.7409602999687195, -1.072996973991394, -1.0865172147750854, -1.2040519714355469, 0.66164231300354, 0.10865118354558945, -0.6002157926559448, 0.25137609243392944, 0.5020433664321899, -0.53029465675354, -0.1172209158539772, -0.3690551519393921, 0.9310470819473267, 1.0178364515304565, 1.2215718030929565, 0.6692854166030884, -0.06564120948314667, 0.577728807926178, 1.346690058708191, -0.08397739380598068, 0.5677359104156494, -0.32090163230895996, -0.21953101456165314, 0.02792138233780861, -1.9149526357650757, -1.0424648523330688, -0.829081118106842, 0.08900155872106552, -0.2258140742778778, 0.6573126316070557, 0.8457332849502563, 0.8357509970664978, 1.8310059309005737, -1.1778934001922607, -0.3643116354942322, -0.46823760867118835, 1.4950259923934937, -0.9039685130119324, -0.3323323130607605, 1.0585018396377563, -1.44578218460083, -0.9869542121887207, -0.06335807591676712, -0.9364471435546875, 1.237905502319336, 1.6059856414794922, 0.11460722982883453, -0.6004586219787598, 1.8554080724716187, -0.8707988262176514, -0.19229519367218018, -0.33915799856185913, 0.10654443502426147, 1.4097986221313477, 0.8850204348564148, -1.5017366409301758, 0.5647249817848206, 1.4583654403686523, 0.867587149143219, -0.1728791445493698, 1.579959750175476, 1.7182042598724365, -0.20082542300224304, -1.1832940578460693, 0.21966442465782166, -1.68339204788208, 0.012822245247662067, 1.112230896949768, -0.23650848865509033, -0.3719457983970642, 1.4402023553848267, 1.2364507913589478, -0.918754518032074, -1.7644575834274292, -0.4476112127304077, 0.3050091862678528, 0.6722472310066223, 1.1604206562042236, -0.012597610242664814, 0.09792593866586685, -0.20576173067092896, -1.225887656211853, -0.30277174711227417, -0.9589043259620667, 0.19970297813415527, -2.0360848903656006, 0.4278497099876404, 1.7207305431365967, -0.9927782416343689, -0.672848105430603, -0.9584586024284363, -0.7865413427352905, 0.6843036413192749, 0.17578504979610443, -0.6281876564025879, 0.4254211187362671, 0.4141641855239868, -0.11088958382606506, 1.0368757247924805, -0.3646939992904663, -0.5846582651138306, -1.0659681558609009, -0.28965282440185547, -0.32512739300727844, -0.08998612314462662, 0.8813266754150391, 0.3940526247024536, -1.9865342378616333, -0.43059760332107544, -0.15118449926376343, -0.9995218515396118, -0.18978814780712128, -0.4636075794696808, -1.2705693244934082, 1.2295706272125244, -0.9145926833152771, -0.46770429611206055, -1.397957682609558, -1.4458478689193726, -0.7410995960235596, -0.5180419087409973, 0.015130986459553242, 0.928919792175293, -0.38442879915237427, 0.04166007041931152, -2.278763771057129, -1.4006444215774536, 0.06874018907546997, -0.9345933794975281, 0.5319766998291016, 1.7143338918685913, -1.9089747667312622, -0.23913562297821045, 0.8993523120880127, -0.866209864616394, 0.8434992432594299, -0.7415388822555542, -1.5469059944152832, 0.10655368119478226, 0.06889011710882187, -1.195109248161316, 1.0010277032852173, -0.7910184264183044, 2.016162157058716, -1.6350769996643066, -0.2534949481487274, 1.5692720413208008, 0.24539901316165924, -0.39318081736564636, 0.3650452494621277, -0.29028740525245667, 0.6104602217674255, 0.44511285424232483, -0.05287439003586769, 0.3040107786655426, 0.33321598172187805, 0.27987924218177795, -0.4412311017513275, -0.8540312647819519, -0.1429014950990677, -0.32130560278892517, 0.132063090801239, 0.6117770075798035, 0.23851804435253143, -0.9444620013237, 0.7832809686660767, -1.5859311819076538, -1.6107382774353027, -0.5432896018028259, 0.0365668348968029, -1.143448829650879, 0.06127737835049629, 0.47929972410202026, 1.100204586982727, -0.14887163043022156, 0.43277308344841003, -0.9628039598464966, -0.011274795979261398, 0.9288668632507324, 0.11687935888767242, -0.7253885269165039, 0.2974155843257904, -0.0809999331831932, -0.7738831043243408, -0.7346991300582886, 0.3655534088611603, -0.1307021677494049, 0.7280544638633728, -0.34140101075172424, -1.7019803524017334, 0.4870230555534363, -0.0980268120765686, -0.5083808302879333, 0.42100799083709717, -0.9761485457420349, -2.6208691596984863, -0.07506164163351059, 0.5054404735565186, -0.6348710656166077, 0.6509073972702026, -0.5784863233566284, -0.37511441111564636, 1.774266004562378, 1.2333463430404663, -1.3763213157653809, -0.9696236252784729, -0.9780712127685547, -0.2256801277399063, -0.4088021218776703, 1.1987897157669067, -1.4409310817718506, 0.35654592514038086, -0.15583497285842896, 0.08608971536159515, 2.48063063621521, -0.6674894690513611, -0.9571633338928223, 0.16109931468963623, -1.0000243186950684, 1.5660072565078735, -1.7447080612182617, 1.4713042974472046, -0.1143866553902626, -0.9389322996139526, 1.4172149896621704, 0.9491737484931946, 0.57033371925354, -0.38254520297050476, 0.8092169761657715, 1.466110348701477, 1.1874159574508667, -0.9647373557090759, 0.7903408408164978, 0.6754348874092102, -1.066611886024475, -1.9212062358856201, -1.1885935068130493, -0.2282857596874237, -0.3021160960197449, -0.18920819461345673, -1.781110167503357, -0.5553500652313232, 0.7875774502754211, 0.6109877824783325, 1.850947380065918, -1.3697491884231567, 0.8392546772956848, -0.35333624482154846, -0.9947064518928528, -0.06407244503498077, -0.6887351274490356, -1.5584092140197754, 0.3669154644012451, 0.28309231996536255, 0.7922171354293823, 1.3671000003814697, -0.5381062626838684, 0.3664284348487854, 1.2593698501586914, -1.1458170413970947, 0.30267810821533203, 1.3333662748336792, 0.20551162958145142, 0.2967936098575592, -0.2394559383392334, -1.9777530431747437, 1.0724540948867798, -0.9978284239768982, -1.287710189819336, 1.343583106994629, 0.4986206591129303, -0.630277156829834, 1.2292184829711914, 0.019492877647280693, -2.0339109897613525, 0.5714581608772278, -0.01789666712284088, 0.5929044485092163, 0.7400689125061035, 1.6113547086715698, -0.43698450922966003, -0.3836107850074768, 1.4940128326416016, 0.16576088964939117, 1.8572665452957153, 1.8340247869491577, 1.4015592336654663, 1.8207290172576904, -0.12945939600467682, 0.39611297845840454, 0.019228138029575348, 0.48142412304878235, 0.2935001254081726, -0.20195446908473969, -2.2613885402679443, 0.19129152595996857, 0.03583439812064171, -1.3344470262527466, -0.5729292631149292, 0.6511067152023315, 2.9262571334838867, -0.4335135519504547, 0.09608162939548492, -0.45366305112838745, -0.05659620836377144, 0.5011752843856812, 0.6237684488296509, -1.035383701324463, 2.341062307357788, 1.7042595148086548, -1.965409517288208, 0.0432925745844841, 0.5974280834197998, -1.0426647663116455, 0.12695226073265076, -0.7394396066665649, 0.46931782364845276, -0.0632254108786583, 1.0178778171539307, -0.5869450569152832, 0.29239657521247864, -1.368280291557312, 0.8402103185653687, 1.027944564819336, 0.17832614481449127, -0.5768421292304993, -0.8996779322624207, 2.403594493865967, -0.3781087398529053, -0.17065875232219696, -1.0521957874298096, -0.750943660736084, 0.7431772351264954, 0.438487708568573, -1.3091753721237183, -0.32150745391845703, -0.457188218832016, -0.6791687607765198, 0.010608645156025887, 0.1123766228556633, -0.44630277156829834, 0.7580958008766174, -0.13415835797786713, -2.992449998855591, -0.9019842147827148, 2.4539859294891357, 1.217844843864441, -0.07324402779340744, 0.5861321687698364, 0.6804134249687195, 0.5713329911231995, -0.9263384938240051, 0.029385659843683243, -0.42287254333496094, -0.7808981537818909, 0.2520962655544281, -1.0261945724487305, -1.4003874063491821, -0.15663471817970276, -0.3952083885669708, -1.2908337116241455, 0.3650526702404022, -0.13493229448795319, 1.0082242488861084, -1.7475649118423462, -0.035863425582647324, -0.5639004707336426, -1.9365769624710083, -0.2839567959308624, -0.39696580171585083, 2.225619316101074, -1.257720708847046, -0.4303229749202728, 2.0295257568359375, 1.135679006576538, -0.513145923614502, 1.4660667181015015, 1.1781271696090698, 0.10360841453075409, 0.37011441588401794, 0.9380621314048767, 0.3197857737541199, -0.728337287902832, -1.5149909257888794, -1.083652377128601, 0.08883102983236313, -0.4370898902416229, -0.3482816219329834, 0.6977394819259644, -2.2470834255218506, 0.9048903584480286, -0.05317709967494011, 0.6058958172798157, 0.08679047971963882, -0.39547738432884216, 0.9707559943199158, 0.1408029943704605, -0.8602501749992371, -0.525962769985199, -1.2942098379135132, 0.4077097475528717, -0.9820181131362915, 2.4522314071655273, -1.0841597318649292, -0.12005222588777542, -0.4443066716194153, -1.7625858783721924, -0.28494468331336975, -0.4348483383655548, 0.18280620872974396, -0.5674709677696228, 0.5362775325775146, -1.0468872785568237, 0.7196658253669739, 0.30509084463119507, 1.805867075920105, -0.2955763339996338, -0.5153407454490662, -0.5829299092292786, 0.6260469555854797, -0.7813149690628052, -0.7946462035179138, 0.7957757711410522, -0.18296264111995697, 0.8149423003196716, 1.4560635089874268, -0.10516176372766495, -0.3519136905670166, 0.35469701886177063, -0.07359638065099716, 1.9521429538726807, 1.3197927474975586, -0.44334590435028076, 1.0836877822875977, -0.687419056892395, -0.5436116456985474, -1.9850273132324219, -0.27306821942329407, 0.6608591079711914, -0.0873088389635086, 0.7975701093673706, -1.357142686843872, 1.6266791820526123, -0.49274009466171265, 0.8301244974136353, 0.9885891079902649, -0.5710000991821289, 0.06954117864370346, -0.15034902095794678, 0.026735836640000343, -0.29388532042503357, 0.4907870292663574, 0.487425833940506, 1.87152898311615, -0.9110029339790344, 0.5576906800270081, 0.6003931164741516, -0.8856748938560486, -0.9166708588600159, 0.7311847805976868, -0.11223447322845459, -1.0259068012237549, -0.9967851042747498, 1.093786597251892, 0.6680031418800354, -0.5845730304718018, -0.7367464303970337, -0.6535777449607849, 0.1962280124425888, -0.014899026602506638, 0.4438236951828003, 1.1878602504730225, -0.17223358154296875, 1.5308035612106323, -0.369090735912323, -0.007387132383882999, -0.2439446896314621, -2.564401626586914, -0.4665440618991852, 0.33851417899131775, -1.9256302118301392, 0.46025076508522034, 0.4941950738430023, -0.8592048287391663, 1.6491644382476807, -1.6508761644363403, 0.7589054107666016, -0.1347128450870514, 1.0943599939346313, 0.25718027353286743, 0.765063464641571, 0.32821130752563477, 1.069950819015503, -0.494875967502594, -1.0532716512680054, 1.2094172239303589, 0.2606086730957031, 1.5338094234466553, 0.24412715435028076, -1.2637242078781128, 0.018483880907297134, 1.6869720220565796, -0.8438755869865417, -1.038853406906128, -1.2801858186721802, 0.5765261650085449, -1.470536231994629, 1.92098867893219, -0.6553360223770142, 1.5619165897369385, -0.3981727957725525, 0.7179068922996521, -0.8449640274047852, -1.5530247688293457, 0.6034660339355469, 0.9380610585212708, -1.255925178527832, -0.3600037097930908, -0.6691641211509705, 0.488444983959198, -0.21127651631832123, -1.1280449628829956, 1.93894624710083, -0.006443407852202654, -1.095051884651184, -0.1673436462879181, -0.6855015754699707, 0.28884097933769226, -0.9397267699241638, -0.13684619963169098, 0.7231183052062988, 0.8514493107795715, 0.07417550683021545, -0.7424779534339905, 0.1688881367444992, -0.6659332513809204, -0.7417624592781067, 1.1665308475494385, 0.09399086236953735, -0.22142243385314941, -0.9699167013168335, -0.4557930529117584, 1.7349566221237183, 0.14768263697624207, 0.29588034749031067, -0.2844827175140381, -0.5104154348373413, -0.16594350337982178, 1.0887655019760132, 0.2525695562362671, -1.1294726133346558, -1.1298742294311523, 0.0005108133773319423, 1.0728503465652466, 0.1866399198770523, -2.6758337020874023, 0.11600644886493683, 0.9567406177520752, 0.5875226855278015, -0.9079678654670715, -1.0985383987426758, -0.915614128112793, -0.5260211229324341, -0.33729100227355957, 0.2207411825656891, 0.6706482768058777, -1.2026678323745728, 1.805031180381775, -0.7212558388710022, 0.07908238470554352, -1.4964138269424438, -0.24831601977348328, 0.32037943601608276, -0.7302899360656738, -1.1444776058197021, 0.3485734462738037, -1.0529861450195312, 1.2248073816299438, 0.2708447575569153, 3.057673454284668, -0.7887944579124451, 1.2095296382904053, 0.8213689923286438, -0.2639428675174713, 0.43599385023117065, -0.10919113457202911, -1.6655062437057495, -0.24099566042423248, 1.2228833436965942, 0.6055814623832703, 0.7167977690696716, -0.42744672298431396, -0.7593178749084473, 2.066250801086426, 1.3349997997283936, 1.8723920583724976, 1.397316575050354, 0.15361960232257843, -0.6352663636207581, 0.3298899531364441, 0.9310998320579529, -0.7687965631484985, -0.34513556957244873, 2.0301673412323, -2.0831899642944336, -1.810799241065979, 0.36363300681114197, 0.20772293210029602, 0.3411499261856079, 1.4359508752822876, 0.7547122836112976, -0.8059055805206299, 0.21760194003582, 0.9006737470626831, -2.1276192665100098, -1.8882321119308472, 1.16444730758667, 0.3785701394081116, 1.0145148038864136, -1.3526490926742554, 0.5639815330505371, -0.5822742581367493, 0.7218924164772034, 0.19273383915424347, -1.3161617517471313, 0.18587644398212433, -0.388092577457428, 0.7414640784263611, -1.691845417022705, -1.185080647468567, 0.8407217860221863, 0.7900581955909729, 2.1847784519195557, -1.2041224241256714, 1.2646076679229736, 0.36395254731178284, 1.9376689195632935, 0.07202121615409851, 0.47547605633735657, -0.6099968552589417, 0.49353644251823425], [-0.19304010272026062, 1.2262334823608398, 0.19389045238494873, 1.0596075057983398, 0.5777371525764465, -0.3628464341163635, 0.2227088063955307, -1.4887548685073853, -0.3377338945865631, 0.3919311463832855, 0.6697924733161926, 0.5013272762298584, -0.1056225523352623, -1.359864354133606, 1.4018566608428955, 0.30608439445495605, -0.8332059979438782, -0.9246636629104614, -1.1568844318389893, -1.5066583156585693, -0.6954651474952698, 2.0102198123931885, -0.5130107402801514, 0.17632289230823517, 0.8233493566513062, -0.4773973822593689, 0.42906373739242554, 1.4764171838760376, 2.3503448963165283, 0.5241636633872986, -1.7346457242965698, 0.7976425290107727, 0.015608425252139568, 0.16544784605503082, -0.5571412444114685, -0.3433198034763336, 2.6093993186950684, -0.45391368865966797, 0.45843324065208435, 1.8830674886703491, 0.2536389231681824, 0.8008847832679749, 0.8913609385490417, -0.9118099808692932, -0.12445040047168732, 0.378243625164032, 1.9475736618041992, -1.3017433881759644, -0.5598869323730469, 0.7582355737686157, 0.5119346380233765, 0.20761188864707947, -1.414484977722168, 0.30534327030181885, -0.882276713848114, -0.19200962781906128, 2.2225494384765625, -0.2218446582555771, 0.0058218939229846, -2.8316714763641357, 1.8993779420852661, 0.2535388171672821, 1.2119882106781006, 0.043575968593358994, -0.0793592557311058, -0.5297849774360657, 1.095376968383789, 1.4723843336105347, -0.16383634507656097, -1.1614753007888794, 0.2738114893436432, -0.6446733474731445, -0.03786935657262802, 0.01850171759724617, 1.2190477848052979, -0.8346247673034668, -1.932760238647461, -0.3608531057834625, -1.6236881017684937, -1.5019737482070923, 0.9738066792488098, 0.08058705925941467, -1.7323626279830933, 0.0700395256280899, 1.159949779510498, -0.6788704991340637, 1.7804522514343262, 0.5339269638061523, 0.2681882977485657, 0.4182720184326172, 0.33791694045066833, 0.5790160894393921, -0.540675699710846, -1.5613059997558594, -0.6095532774925232, -0.21328432857990265, 0.5043886303901672, 0.40791529417037964, 0.1864721029996872, -0.20820806920528412, -0.5200291872024536, 1.5835226774215698, -0.4037684202194214, -0.5354714393615723, -1.9469704627990723, -0.10117156058549881, 0.5531941652297974, 1.7350322008132935, 1.7866346836090088, 0.34294891357421875, 1.1808021068572998, -1.6115177869796753, -0.9811986088752747, 0.31164640188217163, 0.5588271617889404, 0.3949226438999176, 0.18408097326755524, -0.035469505935907364, -0.43054503202438354, 0.01314147375524044, 0.07473848015069962, 0.1974104344844818, 0.843321681022644, 0.1713353991508484, 1.3885895013809204, 0.6723973155021667, 1.0037888288497925, 2.4026424884796143, 0.06404881924390793, -0.6425833106040955, -0.8343533873558044, -0.01165732741355896, 0.4387021064758301, -0.5575720071792603, -0.6629589200019836, 0.7210888266563416, 0.6677293181419373, 0.753553032875061, 0.5430279970169067, 0.8585047721862793, 0.5904057621955872, -0.6544404029846191, 0.41804760694503784, -0.6894963979721069, 1.2424739599227905, -0.6272228956222534, -0.8057335615158081, 0.8797918558120728, -0.09058649837970734, -0.3494453728199005, -0.22173619270324707, 1.0038747787475586, 0.7621011137962341, -0.5732729434967041, -1.779794692993164, -0.27277421951293945, 0.7584167718887329, -1.62772798538208, -2.8400466442108154, 1.8647187948226929, 0.8234792947769165, 0.17781513929367065, 0.5137826204299927, 0.7298119068145752, 0.277140736579895, 0.7289026379585266, -1.0261003971099854, 0.014706629328429699, 0.670207679271698, 1.4221210479736328, 0.8181875348091125, 0.6817073225975037, -0.6284014582633972, 1.5762689113616943, 0.4146899878978729, -0.1948501467704773, 0.5109947323799133, -0.5370987057685852, 0.45671284198760986, -0.5926598310470581, 1.2184829711914062, 0.9339399337768555, -1.1956233978271484, 1.4448573589324951, 0.2718399465084076, -0.860163688659668, 0.8033658862113953, -0.8317272067070007, -0.0037657623179256916, 0.8740005493164062, -0.0023625565227121115, 0.5420148372650146, -0.6536983847618103, -2.669950008392334, 1.1088943481445312, -0.6137424111366272, -0.7848898768424988, 0.691760778427124, -1.333827257156372, -1.4383914470672607, -0.6288925409317017, -0.17215201258659363, -1.3006632328033447, -0.3213226795196533, 0.07451310008764267, -0.44944286346435547, -0.01815125346183777, 1.3432867527008057, 1.243290662765503, -1.0451668500900269, -1.0200014114379883, 0.895351767539978, -0.5791895389556885, -0.07420765608549118, -1.336808443069458, 0.40876713395118713, -0.05599292367696762, -1.5022289752960205, -0.25602987408638, 0.2856288254261017, 0.2361382097005844, -1.8506848812103271, -0.8686364889144897, -0.999603807926178, -0.04826609045267105, 0.2116413712501526, 1.342086911201477, -2.0679972171783447, 0.4557371735572815, 0.6231420040130615, 0.7442777752876282, -1.9810247421264648, 1.166611671447754, 0.875073254108429, 0.10305418074131012, 1.242686152458191, -0.4667556583881378, 0.3188963234424591, 1.9335335493087769, -0.26299595832824707, 0.15365023910999298, -1.0051127672195435, 0.23550967872142792, -0.5654449462890625, 0.43402016162872314, -0.23686328530311584, -0.6213414669036865, -0.5071038603782654, -0.09882871061563492, -0.3876488506793976, 0.07528584450483322, -0.768664538860321, 0.841921329498291, 1.489805817604065, 0.25137728452682495, -0.7282006740570068, -1.6301136016845703, -0.6004891395568848, -0.9910474419593811, 0.7482544183731079, -0.02133297547698021, -1.2503877878189087, 0.5650123953819275, -0.14645801484584808, -1.0378249883651733, -0.2489510178565979, 0.21493792533874512, -0.1347048431634903, -1.6362857818603516, -2.116513967514038, 0.12385952472686768, -0.2980256676673889, -0.705663800239563, -0.9976711869239807, 1.2457551956176758, 0.15668681263923645, -0.9376996159553528, -1.0994501113891602, 0.6086747050285339, 1.2826091051101685, -1.924243450164795, -1.534587025642395, -0.9649402499198914, 0.008187160827219486, -2.399571657180786, 0.4889862537384033, 0.20330055058002472, 0.9807170629501343, 0.9521040916442871, -0.4899142384529114, 2.178297758102417, 1.727950930595398, 1.166280746459961, 1.9311915636062622, -1.4947233200073242, -0.3679807782173157, -0.5664878487586975, 1.5690860748291016, -0.6206238865852356, 0.40340209007263184, 1.3609272241592407, -0.3404232859611511, 0.22693265974521637, 0.8972266316413879, 0.4427106976509094, 0.11003006249666214, -0.5500563979148865, -0.9244303107261658, 0.5210469961166382, 0.07866869121789932, -1.3193694353103638, 0.8718568086624146, -0.36771661043167114, 0.8452725410461426, 0.04882878065109253, 0.0007328607607632875, -0.8600596189498901, -0.8427544236183167, -3.7715187072753906, 1.3297487497329712, -0.077737957239151, -0.9501807689666748, 0.3378811478614807, 0.8529031276702881, -0.3688308894634247, -0.8450905680656433, 1.183423638343811, -0.19791696965694427, -1.6083641052246094, -1.6284652948379517, 1.1974807977676392, 1.0800144672393799, -0.5522832274436951, -2.2976818084716797, 0.016650976613163948, -0.402608722448349, 0.031493768095970154, -0.8263196349143982, 0.7121609449386597, -1.6617966890335083, -1.1297029256820679, 1.6561660766601562, 0.833851158618927, 0.4494923949241638, 0.027744241058826447, 1.6073980331420898, -0.756034255027771, 1.5704199075698853, 0.5636712312698364, 0.3680211305618286, 1.187097191810608, -0.018092403188347816, 1.6223180294036865, -0.3315778076648712, 0.295628160238266, -1.530102252960205, -0.23112370073795319, 0.47752535343170166, 0.6300322413444519, -0.3315901756286621, 0.9766481518745422, 2.5693018436431885, 0.5607831478118896, -0.5564057230949402, 0.3111889362335205, 1.5304840803146362, -0.45424407720565796, -0.5037478804588318, -1.0627646446228027, -0.4898492991924286, -1.1912643909454346, -0.6944172382354736, -2.269901752471924, 1.5461288690567017, -0.5992071032524109, 0.05290086939930916, 0.3517226278781891, 1.6946290731430054, -0.1820230782032013, 1.303133487701416, 1.5894958972930908, 1.0421816110610962, 0.5777643918991089, 1.452483057975769, 1.350754737854004, 1.9863576889038086, -1.0708296298980713, 1.2730493545532227, -0.5410816073417664, -0.3731195628643036, -0.7192315459251404, 1.327656626701355, 1.1566884517669678, 1.0634396076202393, 0.00581086752936244, -2.2107348442077637, -1.8916690349578857, -0.8376793265342712, 0.6447342038154602, -0.614129364490509, -1.3192477226257324, 0.5894110798835754, 1.1402226686477661, 1.0796564817428589, -0.7422325611114502, -0.683190643787384, -1.224313735961914, 0.5160972476005554, 0.16542157530784607, 1.1188768148422241, 0.17748554050922394, 0.02655964158475399, -0.4810877740383148, 0.6206580996513367, -0.46925196051597595, -1.4376872777938843, -0.47280824184417725, -0.6606956124305725, 0.43287283182144165, 1.2670050859451294, -0.5413534641265869, -0.4397101104259491, 0.2197258025407791, 0.42240476608276367, -0.3683847486972809, -1.355296015739441, 1.8128374814987183, 0.25961974263191223, -1.6197049617767334, -1.2384151220321655, -1.2237164974212646, -1.3807941675186157, -0.7416378259658813, 0.9843228459358215, 1.273842215538025, -1.5929862260818481, -0.8891333937644958, -1.5096946954727173, 0.14472967386245728, 1.5350265502929688, 0.9333979487419128, -1.2077869176864624, 0.7423299551010132, -0.37302085757255554, 1.1229934692382812, -0.29966676235198975, -0.0770706757903099, 0.5413399934768677, -0.4035344123840332, -0.13847750425338745, -0.8828397989273071, 0.1988653540611267, -0.6036043763160706, 1.6297765970230103, -0.20334887504577637, -1.2103747129440308, -0.13939885795116425, -0.8782497048377991, -1.103277564048767, 0.7354723811149597, -0.2487330585718155, -0.25591152906417847, 0.1418004184961319, 0.41288259625434875, 0.5487558841705322, 0.8605461716651917, 1.877953290939331, 0.5626299381256104, -1.814117670059204, -0.7282525300979614, -0.2816731631755829, 0.1926124542951584, -0.7562921643257141, -1.45255446434021, 0.27790629863739014, 1.333258032798767, 1.01272714138031, 0.5881571769714355, 0.45832738280296326, 0.8148179054260254, -1.4086520671844482, -0.7076316475868225, 0.6632972359657288, -0.32513749599456787, -0.6699755787849426, -0.6868410110473633, 1.4589084386825562, -0.914891242980957, -0.5751615166664124, 0.7797287702560425, 0.385903924703598, -0.3964994549751282, -0.9116973876953125, 0.630074679851532, -0.39957669377326965, -0.6103789210319519, 0.09681535512208939, 0.17010116577148438, -0.5445776581764221, 2.161034345626831, 0.10062596946954727, 2.0390095710754395, 0.2865592837333679, 0.019158247858285904, -0.5438379049301147, 0.6957257390022278, -0.366317480802536, 0.4786163866519928, 0.8848720192909241, -0.010255113244056702, -0.5516325831413269, -2.005396842956543, 2.500248670578003, 0.5542102456092834, -0.30420878529548645, 0.5613575577735901, -1.536521315574646, 0.5531868934631348, 1.0316128730773926, 0.5734344124794006, -0.7055462002754211, -0.5737525820732117, -1.782564640045166, 0.7944285869598389, 0.18763619661331177, 0.36980363726615906, -1.3542118072509766, 0.13651464879512787, -0.2404560148715973, 0.49665746092796326, 1.073140263557434, -0.9070404767990112, 0.15150386095046997, -0.05560537055134773, -1.831903338432312, -1.0213894844055176, 0.6154099702835083, 0.705206036567688, -1.665748953819275, 1.1939146518707275, -0.279796302318573, 0.19002565741539001, 0.03941689431667328, -1.3421499729156494, 0.13043232262134552, 0.510117769241333, 0.28402283787727356, 0.8140289187431335, -0.4962884485721588, -0.9991921782493591, 0.716536283493042, -0.9760567545890808, 1.5373890399932861, -1.0596710443496704, 1.8911101818084717, 0.7555049061775208, -0.20064350962638855, 0.7775128483772278, -0.28789326548576355, 1.458447813987732, -0.19511958956718445, 0.35054150223731995, -0.9064098596572876, 0.713990330696106, -0.08070863038301468, -2.5839805603027344, -1.9560744762420654, 2.102348804473877, 1.3501476049423218, -0.4144143760204315, -0.4256832003593445, 0.07720106095075607, -0.11022087931632996, 0.606669008731842, 0.5898017883300781, -0.15936508774757385, 0.9363824725151062, 0.8540074825286865, -1.1603628396987915, -0.09662675857543945, -0.7832239866256714, 0.438694566488266, 0.08041726797819138, -0.613854169845581, -0.8284745216369629, -0.8534271717071533, 2.032424211502075, -1.016301155090332, -0.6968938112258911, -0.3770042955875397, -0.1713109314441681, 0.22669395804405212, 0.8890435099601746, -0.22565583884716034, 0.9485495090484619, -0.8380594253540039, 1.330565094947815, 1.149029016494751, 0.6208991408348083, 1.5222740173339844, -0.5682726502418518, 1.2869458198547363, -0.8802956342697144, -2.034268617630005, 1.1961333751678467, 1.1012578010559082, -0.18853385746479034, -0.5921761393547058, 0.6158902645111084, -0.2290167212486267, -0.05519349128007889, 1.816009283065796, -0.8765116930007935, 1.3375697135925293, -0.9409053921699524, -0.6497547626495361, -1.5542465448379517, 1.169728398323059, -0.5713738799095154, 0.9028833508491516, 0.483903706073761, -1.0348774194717407, 1.3146226406097412, -0.33969396352767944, 1.108104944229126, -0.6142849326133728, -1.2292907238006592, -1.4284933805465698, 0.9635088443756104, 0.4938530921936035, -1.191702127456665, -2.8211920261383057, 0.024579232558608055, -0.43228527903556824, -0.15757234394550323, 0.18034011125564575, 0.18919791281223297, -1.9039281606674194, 0.21491874754428864, -0.8683775067329407, 1.1062592267990112, -0.3112713098526001, 0.9029130339622498, 0.13609129190444946, -1.0392115116119385, 0.1875583678483963, 1.6564275026321411, 1.1937432289123535, 0.06716858595609665, 1.048977017402649, 0.5746760368347168, 2.0895729064941406, -0.008352240547537804, 1.5317946672439575, 0.8791451454162598, -0.8470224142074585, 1.5862985849380493, 2.4633426666259766, -1.4586503505706787, 0.8630090355873108, -1.3590266704559326, 0.3503968119621277, 1.1604974269866943, 0.8405215740203857, -0.6799189448356628, 0.557487428188324, -1.0388332605361938, 0.1937284916639328, -0.12626580893993378, 0.1048366129398346, -0.6607257723808289, -0.2926432490348816, -0.9856155514717102, -1.6693882942199707, -1.9754114151000977, 0.5121181607246399, -0.11643365770578384, -0.9756880402565002, 1.3023178577423096, -0.6616513133049011, -0.675263524055481, 0.5195386409759521, -0.9989997148513794, 0.3660115897655487, -0.393739253282547, -2.155048131942749, -0.8579676151275635, -0.37687501311302185, 0.29554009437561035, -0.4207744300365448, -0.5661406517028809, -1.494646668434143, 0.36049774289131165, -1.4597223997116089, 0.8440872430801392, -0.6689254641532898, -0.4665876030921936, 0.6867276430130005, -0.9436796307563782, 0.46959465742111206, 0.5496726632118225, -0.785300076007843, 0.5747516751289368, -0.5068029761314392, -0.7940312027931213, 1.6879775524139404, 2.829613447189331, 1.7256144285202026, -0.8371427059173584, -0.20931591093540192, 0.08817163109779358, -1.0004143714904785, 0.48927101492881775, 1.5580693483352661, -0.1084861233830452, -0.988254725933075, -0.08375386148691177, -1.2990626096725464, -0.18252453207969666, 2.138528347015381, -0.396100789308548, 0.024762891232967377, -0.3727085292339325, -0.27625295519828796, 0.9011101126670837, -1.0933113098144531, 0.40179696679115295, -0.165768101811409, 1.150602102279663, -0.25705060362815857, 0.23070673644542694, 0.5212736129760742, -1.372170329093933, 0.28209900856018066, -0.7352214455604553, -0.5667823553085327, 0.5382959246635437, 0.687162458896637, 0.2270105481147766, -0.0026545317377895117, -0.39520731568336487, 1.6196876764297485, -0.40684497356414795, 0.5308476686477661, -1.1389802694320679, -0.2253694087266922, -0.8808391094207764, 0.2990160584449768, 0.5820603370666504, -1.1238980293273926, 0.649376392364502, 0.816935658454895, -0.22081241011619568, 0.9956021308898926, 1.1503311395645142, -0.5224745273590088, 1.6769572496414185, 0.6143426895141602, 0.5024913549423218, 0.6503615975379944, 0.47719794511795044, -1.4135602712631226, -0.6524832248687744, -1.186951994895935, -0.34282293915748596, 0.27262425422668457, -0.6578336358070374, -1.0803327560424805, -0.6171023845672607, -0.6315843462944031, -1.2277230024337769, 0.16155779361724854, -1.0351581573486328, -0.6122443079948425, 0.9767515659332275, -1.2167420387268066, 0.21079140901565552, -1.8301204442977905, -0.3307592570781708, -2.473130464553833, -0.349623441696167, -0.4955890476703644, 1.8974946737289429, 0.2987212538719177, 1.6783928871154785, -0.3895474076271057, -2.3463850021362305, -2.2612178325653076, 0.12335829436779022, -1.8099626302719116, -0.8433089852333069, 0.7870352864265442, -1.2771670818328857, -0.3568113148212433, 0.18681083619594574, -0.8370718955993652, -1.5208497047424316, 0.59797203540802, -0.688606321811676, 1.6645677089691162, -0.40476199984550476, 1.7081050872802734, -0.09078280627727509, -0.1253557652235031, -1.5873855352401733, 0.6719231605529785, -1.3062251806259155, 0.8618483543395996, 0.4500172734260559, 0.521937370300293, -0.6077442169189453, 0.4281550347805023, 0.7021325826644897, -0.7689209580421448, 0.6340440511703491, -0.8307591080665588, -1.290052890777588, -0.9027007818222046, 1.9460912942886353, -0.16413180530071259, 0.3554486632347107, -1.072986125946045, -1.2341617345809937, 1.644429087638855, -1.239006757736206, 2.2657928466796875, -1.0687769651412964, -1.6833471059799194, 0.2857550382614136, 1.2214610576629639, 0.3705974817276001, 0.5790571570396423, 0.497142493724823, 0.7659957408905029, 1.0038800239562988, 0.6016993522644043, 0.07342735677957535, 1.5117263793945312, -0.48221996426582336, 0.2565365135669708, -0.11413203179836273, -1.2213023900985718, -0.4361557066440582, 1.609742522239685, -0.9693515300750732, 0.7415962815284729, -0.07276441156864166, 0.4120728373527527, 0.705571174621582, 0.8549249768257141, 1.4273579120635986, -0.21034498512744904, 1.0633304119110107, 0.7853227853775024, 0.8714326024055481, -1.9124149084091187, -0.13905476033687592, -0.05051189288496971, -0.3405199944972992, 0.045036304742097855, -0.24584409594535828, -1.7850955724716187, -0.9255926609039307, 0.5127752423286438, 0.46891120076179504, 3.629643440246582, -1.4391534328460693, 1.0223692655563354, 1.070657730102539, 0.1847972720861435, 2.9056155681610107, 0.17580191791057587, -1.1922842264175415, 2.089980125427246, 0.29566690325737, 0.15120579302310944, 0.4478134512901306, 0.8121045827865601, 0.16064868867397308, -0.22206395864486694, -0.9193171262741089, 0.7284061312675476, -0.6643641591072083, 0.1279081106185913, 0.25594547390937805, -0.32417798042297363, 0.2557305097579956, -0.5448362231254578, 0.5466815829277039, 0.00019385981431696564, -0.07700986415147781, 0.8064672350883484, 0.01027179230004549, 0.6907359957695007, -0.3459402620792389, 0.6993528604507446, -0.9005626440048218, -1.7993212938308716, 0.5279461741447449, -0.23461923003196716, -0.42928797006607056, -0.9799599051475525, -0.24821104109287262, 0.7866721153259277, -0.49901700019836426, 0.9519856572151184, -0.3334536850452423, 0.5734350085258484, 0.7809897661209106, 0.5216912627220154, 0.6064348816871643, 0.25417545437812805, 1.1296766996383667, 1.5670188665390015, -0.3265414237976074, 1.5273680686950684, -0.8860862851142883, -1.7101281881332397, 0.3124178946018219, 0.31959325075149536, 1.3128637075424194, -0.2981024980545044, -1.2905921936035156, -0.47313785552978516, 1.2020387649536133, -0.3110087811946869, 0.9251770973205566, -0.4582039713859558, 2.2098610401153564, 1.1340420246124268, 0.8417548537254333, -1.2835367918014526, -0.15047979354858398, -0.90047287940979, 1.3139265775680542, 1.4675463438034058, 0.37183281779289246, 0.738676130771637, 0.5800334215164185, -0.092123843729496, 0.09653370082378387, 1.7433192729949951, 0.10703826695680618, -0.5737418532371521, -0.1404142677783966, -0.4807572364807129, -0.8687192797660828, 1.4587409496307373, -0.06897573173046112, -0.39233145117759705, 2.1412906646728516, 0.02560480311512947, -0.045699093490839005, -1.677139163017273, 1.034574031829834, -0.2511949837207794, 0.7821218967437744, -0.9308178424835205, -0.906951904296875, -0.9767630696296692, -1.0928380489349365, 1.368467926979065, 0.05919305607676506, 0.6426408886909485, -2.5489108562469482, -1.226281762123108, 1.2789297103881836, -1.462910771369934, -0.014480621553957462, 0.19558583199977875, 0.22026072442531586, 1.119147777557373, -0.3173369765281677, 1.1197497844696045, -0.406810462474823, -0.14816579222679138, -0.49863454699516296, -0.1822873055934906, 2.037853956222534, -1.0817219018936157, 0.47688716650009155, 0.40491488575935364, 2.0484344959259033, 1.554061770439148, 0.643979549407959, 1.4131256341934204, -0.9984831809997559, 1.805206060409546, -0.5846437215805054, -0.23859991133213043, 0.32689860463142395, 0.16799485683441162, -1.617982029914856, 0.2590593099594116, -0.4302469491958618, -0.3816758990287781, -1.3099849224090576, 1.1689542531967163, -1.4991918802261353, 2.792012929916382, 1.7802215814590454, -1.3485443592071533, -1.0679889917373657, -0.0715167447924614, 2.111090898513794, 1.2256076335906982, -1.1069244146347046, -0.029643232002854347, -0.30970335006713867, 0.5062976479530334, 0.4051986336708069, 1.420467734336853, -0.9023495316505432, 1.0126621723175049, -0.11561557650566101, -1.9235246181488037, -0.2769518196582794, 0.9097902774810791, 1.097011685371399, -0.10274140536785126, -0.13721151649951935, 0.05747973173856735, 0.572696328163147, -1.3296730518341064], [0.5904775857925415, 1.7046281099319458, -0.27819550037384033, 0.9334526658058167, 1.3204964399337769, -0.0442899651825428, 1.3134465217590332, -0.36363640427589417, -0.3129533529281616, -0.15595708787441254, 1.3005304336547852, -0.23661299049854279, 0.5387893319129944, 1.438245415687561, 0.5888455510139465, -1.334221363067627, 0.8857679963111877, 0.057651083916425705, -0.4441724121570587, 0.2991635799407959, -0.05990499258041382, -1.0629969835281372, 0.013441335409879684, 2.554265260696411, 3.416289806365967, -0.5043772459030151, 1.1544554233551025, -1.6487724781036377, -0.4335255026817322, 0.8432113528251648, -0.6751831769943237, -2.0599491596221924, 2.514507293701172, -0.17414912581443787, 0.24981431663036346, 2.006108283996582, 0.05387242138385773, -1.5645768642425537, 0.46429648995399475, 1.0888960361480713, 0.10820160061120987, 0.6999030709266663, -0.6193181872367859, -0.08739829063415527, -1.441180944442749, 0.06325693428516388, 0.9442735910415649, 0.7794701457023621, -0.8911929726600647, 1.0772836208343506, -0.41040703654289246, -0.8890120983123779, -0.47941339015960693, -0.6772974133491516, -1.2344696521759033, 1.1167923212051392, 0.6429552435874939, -0.4682333171367645, -0.5713140368461609, 1.4503511190414429, 0.07816626131534576, 0.27515530586242676, 0.08547692745923996, 1.1166669130325317, 1.5655487775802612, -0.9492658972740173, 0.6339871287345886, -0.8970606923103333, 0.7623485326766968, 0.0030072431545704603, 0.3924754559993744, -1.3663737773895264, 1.041379690170288, 1.5687291622161865, 0.5146608352661133, 0.21833349764347076, -0.40342646837234497, 1.2537676095962524, -1.1998804807662964, -0.8124983906745911, 0.9633751511573792, -1.615391492843628, 0.8254452347755432, -0.7054536938667297, 0.34227070212364197, -1.2379106283187866, -0.7777390480041504, -1.7645275592803955, 1.4889512062072754, 0.32015514373779297, 0.3586503863334656, -1.7689944505691528, -0.7730910181999207, -0.13054484128952026, 0.7474555373191833, -1.921055793762207, -0.07053481787443161, -0.16268786787986755, 1.3176264762878418, 0.41329294443130493, 0.13223491609096527, 0.7867686748504639, -0.4714169502258301, 2.208540201187134, -1.1491572856903076, 0.8903343677520752, 0.04317040368914604, -0.7964937686920166, -0.20371858775615692, -1.878751277923584, -1.854753851890564, -0.052369825541973114, -0.3626035153865814, 1.2769914865493774, 0.7192100286483765, -0.7529417276382446, 0.5060669779777527, 0.10934658348560333, -1.4473793506622314, 0.24622298777103424, -0.5598650574684143, 0.1345660388469696, -1.238796353340149, -0.38190150260925293, 0.7059584856033325, -0.7942852973937988, -0.4385260045528412, 1.690517783164978, -0.5426547527313232, 0.6371685862541199, 1.2786612510681152, 0.73292076587677, 0.5972516536712646, -0.2275266945362091, -0.2176564335823059, -0.6327296495437622, -1.8162167072296143, 0.37607207894325256, -0.18786752223968506, -0.4031289219856262, -0.592829167842865, -1.2694591283798218, 0.4800449013710022, 0.5564231872558594, 0.6073122620582581, -1.2551159858703613, -0.4453107416629791, 0.09723786264657974, 1.1728050708770752, 1.0660345554351807, -1.0006824731826782, 0.9886482954025269, 0.1546151041984558, 0.3225370943546295, -0.31722909212112427, 0.5265786647796631, 1.7149975299835205, 1.4975301027297974, -0.5312267541885376, 2.4465582370758057, 0.18646495044231415, -0.17029935121536255, 0.5825906991958618, 0.9076058268547058, -0.20700007677078247, -0.1414826363325119, 2.173065662384033, 1.4532592296600342, 0.08210067451000214, -0.9434627294540405, 0.08869839459657669, -0.597216010093689, -0.5380755662918091, -1.0838052034378052, -0.8072226047515869, 0.15172125399112701, 0.7568307518959045, 0.7871950268745422, 0.10861436277627945, 1.780413269996643, -0.471675843000412, 1.729116678237915, -0.2869942784309387, 2.1675894260406494, 0.24805977940559387, -2.5479233264923096, 0.28007349371910095, 2.4628655910491943, -0.5339403748512268, -1.633109450340271, -0.6308245658874512, -1.628176212310791, -0.06166469678282738, -0.623469889163971, 1.0131845474243164, -0.04745868220925331, 2.326496124267578, 0.7518987059593201, 0.562995195388794, -0.5327471494674683, -0.6811547875404358, 1.3512662649154663, -1.4108977317810059, 0.0018616729648783803, 1.764202356338501, 0.7960123419761658, 2.0705056190490723, 0.7515701055526733, 1.4350903034210205, 2.2671453952789307, 0.46592262387275696, -0.33389589190483093, 1.793496012687683, -1.052871823310852, -0.03570360317826271, -1.1576309204101562, -0.44822466373443604, 0.7894232273101807, -0.33609840273857117, -2.038083791732788, -1.1805288791656494, 2.1180779933929443, -0.22861410677433014, -1.354297399520874, -1.3444303274154663, -0.27802127599716187, -0.0658506229519844, -1.331713318824768, 0.8992847204208374, 0.43210098147392273, 1.7489598989486694, 0.4017845094203949, -0.40353623032569885, 0.08358438313007355, 1.085261344909668, -0.39448341727256775, -0.20676478743553162, 0.6103160977363586, 0.7042972445487976, -1.0061368942260742, 0.20662468671798706, -1.2836380004882812, 0.8015792965888977, 1.3047181367874146, -1.244682788848877, 1.142500400543213, -0.8505120873451233, 0.5238234996795654, 0.8330037593841553, -0.3338339328765869, 0.9428049921989441, 1.614330530166626, -0.8219480514526367, 0.10453884303569794, 0.8962520956993103, -1.306218147277832, 0.8428499102592468, 1.5577021837234497, 0.9058266878128052, 1.7799813747406006, 0.8552805781364441, -0.15139175951480865, -0.18203923106193542, -0.2470829039812088, -1.3406678438186646, -1.6794019937515259, -0.9213853478431702, -0.5187825560569763, 0.3959383964538574, -1.290696620941162, -0.9231120944023132, -0.5056407451629639, -1.0699496269226074, 0.3734796941280365, 0.3007862865924835, 1.4475246667861938, 1.0457745790481567, -0.2756539583206177, -0.13566549122333527, 0.3757830262184143, 0.14353494346141815, -1.473814606666565, 0.21392613649368286, 0.726917028427124, -0.9896374344825745, -0.5995203852653503, -0.1216941550374031, -0.588425874710083, 1.9866827726364136, 1.1540383100509644, -0.435992956161499, -0.4244186282157898, -0.36166971921920776, -0.9231866002082825, -1.625639796257019, 0.14506521821022034, -0.5272822380065918, 0.7289782166481018, -1.186328649520874, 0.15005706250667572, 0.15649370849132538, -0.4665371775627136, 0.31475526094436646, 0.9601387977600098, 0.1498233824968338, -0.005199036095291376, -0.31749048829078674, 0.996060848236084, -0.14656908810138702, 0.04122517257928848, -0.1539500653743744, -0.04949618875980377, -0.916289210319519, 0.09445749968290329, 0.9649187922477722, 1.4099764823913574, 0.4898911118507385, -1.5483357906341553, -0.8180423974990845, 0.6310721635818481, 1.0354557037353516, 2.332324981689453, -0.6145235300064087, -1.3323891162872314, -0.011481016874313354, 0.9031535983085632, -1.0949339866638184, -0.7309545874595642, 0.35805878043174744, -0.917984664440155, -0.9390807747840881, 0.08459852635860443, -1.9560459852218628, -0.21052685379981995, 0.253360390663147, 0.6158413290977478, -0.7090267539024353, -2.560765266418457, 0.860403299331665, 2.195384979248047, 0.8540354371070862, -0.5146945714950562, 0.23092104494571686, -0.02053576149046421, -0.07451567053794861, -1.9966551065444946, -0.02662735804915428, -0.21470731496810913, 0.4495674967765808, 0.3883976936340332, 1.992530345916748, -0.7274227738380432, 2.0365498065948486, -2.175057888031006, -0.31551072001457214, -0.4101290702819824, 0.3119155168533325, -0.38675054907798767, 1.1616324186325073, 0.3890281319618225, -0.49659839272499084, 0.7414090633392334, 1.5940810441970825, 1.6157786846160889, -0.25113439559936523, -1.2492784261703491, 1.0227456092834473, -0.10823876410722733, 0.0851365402340889, 0.27070584893226624, 0.7104914784431458, -0.08460534363985062, -1.3129374980926514, 0.038954511284828186, 0.6681613922119141, 0.008527652360498905, 0.4536922574043274, 1.1311556100845337, -0.3241269290447235, -0.4091874659061432, -0.5319418907165527, 0.14330239593982697, 1.7683533430099487, -1.4094854593276978, 1.5989927053451538, 0.371928870677948, 0.8001230955123901, -0.7214111685752869, -0.5513439178466797, 0.34771665930747986, -0.14740285277366638, 0.2728230059146881, 1.706522822380066, 0.5009474754333496, -0.22250857949256897, 0.5571361780166626, 1.2355773448944092, 1.3245716094970703, 0.8809783458709717, 0.5958285331726074, 0.31554338335990906, -0.09478794783353806, 0.6428561210632324, -0.6851530075073242, -0.8266199231147766, 0.169833704829216, 1.7261954545974731, -0.2958490550518036, 0.811787486076355, -1.1656343936920166, 0.9931638240814209, 0.9267598986625671, -1.0689127445220947, 1.0329537391662598, -0.8144237995147705, -0.03266309201717377, -1.5779725313186646, -0.7567744255065918, -0.4147762954235077, 1.5669931173324585, 1.2239915132522583, 0.38918963074684143, -1.4871379137039185, -1.2477797269821167, 0.6381255984306335, 1.7331464290618896, 0.11869260668754578, 0.369546502828598, 0.6885910034179688, -1.1148959398269653, 1.2027618885040283, -0.8672126531600952, -0.020097944885492325, 0.2081192135810852, 0.32665157318115234, -1.045316457748413, 1.468732237815857, 0.4280189275741577, 2.292337656021118, 1.4174978733062744, -0.6550114154815674, -1.9659581184387207, 0.8933612108230591, 0.017196131870150566, 0.04816685989499092, 0.031650397926568985, 0.8287514448165894, -1.647481083869934, -0.047210000455379486, 1.1500705480575562, -0.41245147585868835, 0.0022404107730835676, 0.7413703799247742, -0.4827944040298462, 2.22295880317688, -0.9199857115745544, -1.772072196006775, -2.069751262664795, -0.6895699501037598, -1.4718868732452393, 1.890235185623169, 1.6069179773330688, -0.14103351533412933, -0.4602936804294586, 0.6947502493858337, -0.5501917600631714, 1.1843714714050293, 0.011993689462542534, 0.41446685791015625, 0.12707528471946716, 0.9029202461242676, 0.35210150480270386, 0.08068521320819855, 0.0948110818862915, -0.10117576271295547, 0.2280409336090088, -1.45001220703125, -1.7942049503326416, 1.0493601560592651, -0.11366578191518784, -0.10652238130569458, -0.6649962663650513, -1.7094618082046509, -0.7135909199714661, 0.5851017236709595, -0.810447633266449, -0.3186321556568146, -0.3172781765460968, -1.7057174444198608, -0.287554532289505, -1.286814570426941, -1.022164225578308, -0.06810538470745087, 0.03862255811691284, 0.12511152029037476, 0.5553320050239563, -0.8359215259552002, 2.3728315830230713, 0.16584213078022003, -0.32157060503959656, -1.287054419517517, -0.17822682857513428, -1.2970291376113892, 0.6786596179008484, -0.4029703140258789, -0.2760866582393646, -1.772657036781311, -1.2798715829849243, 1.7662681341171265, -1.598907232284546, 0.5261587500572205, 0.000677051895763725, -0.9140608906745911, 0.4063739776611328, 0.7679518461227417, 0.25650572776794434, -1.4036312103271484, -0.6986969113349915, 0.19780099391937256, 0.55687016248703, 0.8779647350311279, 0.968865156173706, -1.591748595237732, 1.6604399681091309, 1.8354626893997192, -0.7992058992385864, 0.8783413767814636, -0.09711656719446182, -0.7068427205085754, 0.08690323680639267, -1.0057908296585083, 0.208656445145607, 0.6626617908477783, 0.7789855599403381, -0.002615543082356453, -0.13066266477108002, 1.0314891338348389, -1.6896047592163086, 0.2871473431587219, 0.1298864781856537, -1.2236934900283813, -0.7910036444664001, 0.772655189037323, -0.8791540861129761, 0.3390733301639557, 0.6488165259361267, 0.001976973842829466, -0.7333884835243225, -0.7044150233268738, -0.7733356952667236, -0.8450546860694885, 0.9197712540626526, 0.27695074677467346, 0.6658201813697815, 0.45637091994285583, -0.639466404914856, 0.3315049111843109, 2.5158166885375977, -1.1682968139648438, 0.4529200494289398, -0.972978413105011, -0.4765971899032593, -0.9689014554023743, -1.1332091093063354, -0.9112208485603333, -0.16999448835849762, 0.46640193462371826, 1.9384220838546753, 0.5662000179290771, 2.010446786880493, 1.7231080532073975, -2.9302613735198975, -0.7724749445915222, 0.4743361175060272, 0.17422440648078918, 0.3706669807434082, -0.6357808113098145, -0.9968117475509644, -0.08039294183254242, 0.7798861265182495, 0.723980188369751, 0.07630490511655807, 0.4216824173927307, 0.5586754679679871, -0.9708420634269714, -2.5572807788848877, -0.33510228991508484, 0.4437662661075592, 1.2613904476165771, -0.05530865117907524, 0.3854615092277527, -0.6384117603302002, 0.06116246059536934, 0.2714971899986267, 1.0349442958831787, -0.6008129715919495, 1.0631107091903687, 0.7034539580345154, -0.4463886320590973, 0.946406900882721, 0.260978102684021, 0.9519619345664978, -0.5576732158660889, 1.0750632286071777, 0.83409583568573, 0.7599142789840698, 1.0082160234451294, 0.6119426488876343, -0.8665590286254883, -0.3207370340824127, 1.7249926328659058, 1.2675853967666626, -0.1273176372051239, -0.678966760635376, 0.16588251292705536, -0.6558187007904053, -0.3399214744567871, -0.2749776840209961, 0.5663049221038818, -2.610419750213623, -0.5232585072517395, -0.08439904451370239, -0.3655744194984436, -0.5339506268501282, 1.0262538194656372, 0.2773021459579468, 0.20356740057468414, 0.7117359638214111, 0.3228704631328583, 1.5245401859283447, -1.469014286994934, -1.173996925354004, 1.6332594156265259, 0.2841533422470093, 0.9408605694770813, 1.1481833457946777, -1.3938413858413696, 0.653740644454956, 1.9619522094726562, -0.46310561895370483, -0.6455981731414795, -0.4137505888938904, -0.47720372676849365, -0.46849367022514343, -0.24087539315223694, -0.5636074542999268, 0.10585875809192657, 0.9827979207038879, 0.7764446139335632, 1.110319972038269, 1.9850516319274902, 0.14019834995269775, -0.022558847442269325, -1.7004680633544922, -0.9758879542350769, 0.21552196145057678, -1.1619462966918945, -0.4221835434436798, 1.2499951124191284, 0.2749699354171753, -0.026119790971279144, -0.5359388589859009, 0.93294358253479, -0.3807353973388672, -3.4737746715545654, -1.6677932739257812, -1.8941227197647095, 0.9161717295646667, -0.37792089581489563, 0.6300143599510193, 1.7374910116195679, 0.6267340779304504, -0.02212626487016678, 1.3590517044067383, -1.4125032424926758, 2.0783748626708984, 1.1726914644241333, 0.7770204544067383, -0.19347527623176575, 0.1821325719356537, 0.4855654835700989, -0.4943675994873047, 0.9117435216903687, -0.31357672810554504, 0.3584291934967041, 0.9164655208587646, 1.1986850500106812, 1.8007036447525024, -0.7033655047416687, -0.70558762550354, 0.10287752747535706, -1.3268795013427734, -0.8524866700172424, -0.686017632484436, -0.16320395469665527, -0.6509910225868225, -0.29506823420524597, -0.3749457001686096, -0.054736390709877014, -1.094283938407898, -2.548570156097412, -0.545387327671051, -0.6136329174041748, -0.8464294672012329, 1.4385170936584473, 0.22541317343711853, 0.020187338814139366, 0.9289815425872803, 0.2988056242465973, 0.31546345353126526, -1.6354731321334839, 1.0593187808990479, -0.6931518316268921, 1.590814471244812, -0.2974800169467926, 0.18078669905662537, 1.0189087390899658, 0.15719643235206604, 0.28626295924186707, 0.1634616255760193, 0.6871321797370911, 0.0949576124548912, 2.1678104400634766, 1.6385382413864136, -0.539878249168396, -0.3529866635799408, 1.3592970371246338, -0.4386150538921356, -0.6208416819572449, -0.25013047456741333, -1.298527717590332, -0.4555011987686157, 0.13874176144599915, 0.214828222990036, -1.4780062437057495, -0.295354962348938, -0.6262869238853455, -1.432785153388977, 0.5706020593643188, 0.4704212248325348, -0.8614292144775391, -4.609532356262207, -1.136567234992981, -1.0815614461898804, 0.047172315418720245, 0.30020660161972046, 0.3930632472038269, -0.5409142374992371, 3.001967668533325, 0.2807815670967102, 0.5580421686172485, -0.9287754893302917, -0.3344581127166748, 1.5585952997207642, 0.7928950786590576, 0.22815385460853577, 0.9896813035011292, 1.0267665386199951, 1.1856160163879395, 0.06962708383798599, -0.9066847562789917, -0.10826613754034042, 1.0536733865737915, -1.4596487283706665, 1.641998529434204, -0.07029812783002853, -1.7935971021652222, -1.3984616994857788, 2.041537046432495, -0.13595567643642426, -0.8177617192268372, 0.8158089518547058, -1.1387255191802979, 0.9766553640365601, 0.37424352765083313, -1.239644169807434, -0.9363945126533508, 0.7754955291748047, -0.2263290286064148, 0.91194748878479, -0.6599171161651611, 1.570942997932434, -0.05323124676942825, 0.37056371569633484, -0.49165451526641846, -0.15280570089817047, -1.571698546409607, 0.15462031960487366, -1.5557608604431152, 0.06479788571596146, -1.344028115272522, 2.214376449584961, -0.2661161720752716, 0.35228803753852844, -2.8276002407073975, 0.44415777921676636, -1.9968554973602295, 0.2587686777114868, -0.7316559553146362, -1.6451871395111084, 0.3807637691497803, -1.3901159763336182, 1.1101527214050293, 1.2679003477096558, 1.498102068901062, -0.23670560121536255, 1.5938891172409058, 1.4060088396072388, 1.4686120748519897, -0.24738915264606476, 1.2622839212417603, -0.9686210751533508, 0.6181759238243103, 1.5071702003479004, 0.3187675476074219, 1.1921634674072266, -0.9534939527511597, -0.5892545580863953, -1.2542222738265991, 1.4305202960968018, 0.44080087542533875, 0.4447185695171356, 1.4945650100708008, -0.5441794395446777, 1.0075552463531494, 0.9671168327331543, 0.38423779606819153, -0.6484370231628418, -1.3795998096466064, -0.630561351776123, 0.6857591867446899, -0.5861343145370483, 0.10853468626737595, -0.6614980697631836, -0.10300710052251816, 1.423576831817627, 0.532233715057373, -0.7796892523765564, -2.0524678230285645, 0.1287573128938675, 0.5381879806518555, -0.2287844866514206, -0.26201608777046204, 0.8964096307754517, 1.874879002571106, 0.016745438799262047, -0.20184355974197388, 0.5130186080932617, 0.4438709616661072, -1.0948823690414429, -1.7723420858383179, 0.9784632325172424, -0.1081174686551094, -0.7342214584350586, 0.22262656688690186, 0.10433951765298843, -1.1359096765518188, 1.2834113836288452, 0.7145600318908691, -1.3240077495574951, -0.29840177297592163, -1.583616852760315, 0.30272766947746277, -2.0137500762939453, 1.3901325464248657, -0.5730758309364319, -0.3097149729728699, 0.8220649361610413, 0.7204146385192871, 0.2214674949645996, 0.12872806191444397, 0.6084107160568237, 1.7157971858978271, -0.3516365587711334, 0.4397655725479126, 0.32462382316589355, 0.661839485168457, 0.5535516142845154, 0.2836419939994812, -0.006330948323011398, 0.250918447971344, 0.23163266479969025, -2.371079444885254, -2.2432143688201904, 1.1171962022781372, -2.2522037029266357, 0.3288589417934418, 0.17098601162433624, -0.7537410259246826, -0.033122479915618896, 1.8746311664581299, -0.10822168737649918, -0.8311346173286438, 0.6704711318016052, -0.9737725853919983, -0.4494585692882538, 0.20200932025909424, -0.12240821123123169, 1.6057778596878052, 0.9142146706581116, -1.0568612813949585, 0.13332663476467133, 0.5085103511810303, -0.863446056842804, -0.29570257663726807, 0.6037353277206421, 1.176979899406433, -0.5603704452514648, 0.8202121257781982, -0.18211442232131958, -0.8742943406105042, -0.5790966749191284, 0.679468035697937, -0.3255814015865326, -0.19947566092014313, 0.700365424156189, 0.41200077533721924, 0.08297952264547348, 0.217413991689682, -0.24866381287574768, 0.806018054485321, 0.21130584180355072, 0.21724411845207214, 0.12872831523418427, -0.2515665590763092, 0.7496645450592041, -0.8145870566368103, 1.0055252313613892, -0.12988264858722687, 0.978851854801178, -0.8242601752281189, -2.4565372467041016, -0.7781897783279419, 0.6573891043663025, 2.0731372833251953, 1.1230508089065552, -0.2532554566860199, -2.044448137283325, -0.1099867969751358, 0.22345943748950958, 0.15530134737491608, 0.8424087166786194, -2.0872702598571777, -0.285199910402298, 0.33858492970466614, -0.3738485276699066, -0.6066193580627441, -2.1135876178741455, 1.4715025424957275, -0.1872314214706421, 0.3543546199798584, -0.43265727162361145, 0.011990655213594437, 2.31150484085083, 0.35080641508102417, 0.33418646454811096, 0.9497694969177246, -1.053163766860962, 1.1668983697891235, -0.41545575857162476, 1.1330232620239258, 0.5872347950935364, 0.1219741553068161, -0.5261132717132568, -0.42561379075050354, -0.829237163066864, 1.628922939300537, -0.5885570645332336, -0.12342332303524017, 0.29453369975090027, 1.3509913682937622, 0.17220589518547058, 0.572334885597229, -2.8210973739624023, -0.483155220746994, 1.4570033550262451, 0.6870081424713135, -1.0739715099334717, -0.02560579404234886, -1.0518717765808105, 1.049973487854004, -1.2928249835968018, -0.20171909034252167, -0.5929915308952332, 0.2769595980644226, -0.628807008266449, -0.3648760914802551, 1.0048249959945679, -1.0887480974197388, 0.0639273151755333, -0.13754886388778687, -0.8497208952903748, 1.136081337928772, 1.1974680423736572, -0.16286110877990723, -1.1535053253173828, -0.0653616189956665, -0.3753512501716614, -0.45384538173675537, -0.8831372261047363, -1.1120290756225586, -0.8184220194816589, -0.5329917669296265, 1.666952133178711, 0.059092264622449875, -0.44169479608535767, 0.7900453805923462, 0.493808776140213, 0.8058308362960815, -0.3849390149116516, 0.9194617867469788, 1.3945287466049194, -0.22561641037464142, 0.17898769676685333, 0.6329557299613953, -1.680543303489685, -1.4422487020492554, 1.178067922592163, 1.2145670652389526, 0.2463914155960083, 1.1619300842285156, -1.2589367628097534, 0.7365511655807495, 1.5528076887130737], [-2.7159736156463623, -0.8009846210479736, 0.07416166365146637, 0.4310521185398102, 1.7052489519119263, 0.36373287439346313, -1.8851968050003052, 0.1680501103401184, 0.9679111838340759, -0.24303211271762848, 0.5331258177757263, -1.151167869567871, 1.1556133031845093, -0.6104269027709961, -0.6096934676170349, 0.563979983329773, -1.6464637517929077, -0.5937831401824951, 0.73743736743927, -0.6201812624931335, -0.44189828634262085, -0.18616937100887299, -2.605985641479492, -0.02188778668642044, -1.0257229804992676, 0.004155757371336222, 0.6433110237121582, -1.2439470291137695, -2.2478740215301514, -0.608910083770752, -0.43381983041763306, -0.37847304344177246, 1.0416492223739624, 0.025795580819249153, -1.4762693643569946, 0.30512291193008423, 0.4221062660217285, 0.584053099155426, 1.3973766565322876, -0.5197021961212158, 0.6609512567520142, -0.06849995255470276, -0.8590616583824158, 0.18914376199245453, 1.101741909980774, 0.6827086210250854, -0.48721233010292053, -0.42213165760040283, 0.08941403776407242, 1.9059257507324219, 0.20731176435947418, -0.862435519695282, 0.006104099098592997, -0.05395040288567543, 1.186261534690857, 0.4342668652534485, -0.3240208625793457, -1.3549247980117798, 0.15243691205978394, -0.228105828166008, 0.39777976274490356, -0.3061493933200836, 0.9377769231796265, 0.34241369366645813, -1.9304145574569702, 1.2521764039993286, -0.1759026199579239, -1.5741323232650757, 0.158151313662529, -1.886914610862732, 1.05722975730896, 0.4302555322647095, 1.9829139709472656, -1.1851317882537842, 0.17198802530765533, 0.19087620079517365, 0.8905180096626282, -1.173012375831604, -0.6591871976852417, -0.9149479269981384, -0.7577287554740906, 0.5419008731842041, 0.8673335313796997, -0.5170513391494751, 0.7295752167701721, -1.2760645151138306, 0.23130148649215698, -0.3893154561519623, 0.2267642766237259, 0.44714510440826416, 0.8102632164955139, -1.5771821737289429, -0.130656436085701, 1.0519399642944336, -1.9736214876174927, -0.018734818324446678, -0.3243393301963806, -1.8662803173065186, 0.5737867951393127, -0.5540427565574646, 2.285726308822632, -0.13856802880764008, 1.9235608577728271, -0.6077456474304199, -1.806761622428894, 0.023878339678049088, -0.516423225402832, -1.2735424041748047, -0.007832097820937634, 0.6180687546730042, 1.1343539953231812, 0.6881558299064636, 0.09563718736171722, 0.1939527988433838, 0.026511792093515396, 0.5183721780776978, -0.38530704379081726, -0.26903799176216125, 1.0733819007873535, 0.7757916450500488, -0.2832849323749542, 1.3840742111206055, 1.4799940586090088, 0.15001384913921356, 0.15450367331504822, 0.5136322379112244, -2.0529329776763916, -2.189152479171753, -2.3167126178741455, 1.4724934101104736, 0.36129772663116455, 0.4721870422363281, -0.030590280890464783, -0.6488658785820007, 1.6375747919082642, 0.10038702934980392, -1.1326608657836914, -0.8219698071479797, 0.2594447433948517, -0.5503754019737244, 0.31998658180236816, 0.18627475202083588, -0.8315921425819397, -0.4955456554889679, 0.8524784445762634, 0.7739942073822021, -0.3033427298069, -0.8128154873847961, -0.8506912589073181, 0.9418671131134033, 0.2842150628566742, 1.249754786491394, -0.6412156224250793, 0.6843193173408508, 0.5668193101882935, 0.09363777190446854, -1.3695815801620483, 0.4518336057662964, 2.681748390197754, 0.21837924420833588, 2.0355465412139893, 1.4995375871658325, -0.5139536261558533, 0.027037691324949265, -1.383785605430603, -1.5961860418319702, 0.8702751398086548, 0.11922235041856766, -0.48735925555229187, -0.6042611598968506, -0.7495526671409607, -0.1726740598678589, -1.5084890127182007, -1.4264358282089233, -1.3164598941802979, 0.4247948229312897, 0.39767396450042725, -0.8501303791999817, 0.6089696288108826, -1.4161229133605957, -0.7121508121490479, 1.1965943574905396, -0.5355052351951599, -0.35402777791023254, -0.16721908748149872, 0.01962323859333992, 1.0550858974456787, 1.042637825012207, 0.6495868563652039, -0.8184410333633423, -0.3963567316532135, -0.17445603013038635, -1.0091874599456787, 1.8510743379592896, 1.1894944906234741, 0.34587326645851135, 1.2383068799972534, -1.1344575881958008, 0.021779963746666908, 1.5252248048782349, 0.5328428149223328, 0.1453375667333603, -0.7129002809524536, -1.7845520973205566, -1.8400235176086426, -0.910592257976532, -0.0048491088673472404, 0.11815258860588074, -2.842486619949341, -0.2975136935710907, 2.0760483741760254, 0.7076349258422852, 0.3325178027153015, -1.439363718032837, 0.46648237109184265, 0.6477466225624084, 0.37033751606941223, 1.167925477027893, -0.15222540497779846, -1.8561897277832031, 0.14620114862918854, -1.1337970495224, -0.7493941187858582, 0.5484206676483154, -0.17163370549678802, -0.8814366459846497, -1.3459718227386475, -0.37285280227661133, 2.107104778289795, -0.04413904622197151, 1.6186624765396118, 0.996171772480011, 0.5573764443397522, 1.4523940086364746, -0.07798967510461807, 1.5853043794631958, -0.42219486832618713, -0.07239154726266861, -0.9400032758712769, -0.6007009148597717, 0.48811373114585876, -0.6786968111991882, -0.4298807680606842, 0.5590656399726868, 1.4600611925125122, 0.39978718757629395, 0.5372166037559509, 1.3568533658981323, -0.18873833119869232, 1.1283398866653442, 0.5675210952758789, 1.653685450553894, -0.11459308862686157, 1.1396971940994263, 1.2959305047988892, 1.0014601945877075, -1.0711122751235962, 0.7185306549072266, 1.5566295385360718, -0.6655768156051636, 0.02488677389919758, -0.8243628144264221, -1.7090436220169067, -0.4744087755680084, 0.5707998871803284, 2.291374683380127, -1.544904351234436, 1.069128155708313, 0.526266872882843, -2.8173115253448486, 0.40942153334617615, 1.4046472311019897, -0.5673673748970032, 0.6485342979431152, 0.12018220871686935, 0.5560315847396851, 0.8856935501098633, -1.4909900426864624, 0.6571235060691833, 0.8194349408149719, -1.1188782453536987, 0.3679022789001465, -0.5214950442314148, -0.5861018300056458, -0.5471935272216797, 2.879631519317627, 0.6993563175201416, -0.4898008406162262, -0.6606693267822266, 1.3055542707443237, 0.1984250545501709, 0.3446897268295288, 0.3635994791984558, -1.2077112197875977, 0.7467098236083984, 0.9079388380050659, 0.31085869669914246, -1.0603288412094116, -0.3928106129169464, -1.4091514348983765, -0.7585033774375916, -1.558027744293213, -0.24074244499206543, 0.20942813158035278, 1.0418611764907837, 0.6103317737579346, -0.8834729790687561, -0.4475295841693878, 1.0547783374786377, 0.547609269618988, 0.6144314408302307, -0.3444138169288635, -1.2476063966751099, 1.1466177701950073, -0.31144747138023376, 0.8199666738510132, 0.9881192445755005, 1.3558002710342407, -1.5013397932052612, -1.5531443357467651, 0.12832516431808472, 0.1012810692191124, -0.41906213760375977, -1.5093097686767578, -0.8385559916496277, 1.1524171829223633, 1.5939269065856934, -0.514215350151062, 1.751306414604187, -1.2929528951644897, 1.5218504667282104, -0.7173842191696167, -0.6526462435722351, 0.5133340358734131, 0.10787714272737503, 2.0905299186706543, -0.350900799036026, 0.11805697530508041, 0.1991392970085144, 0.7013114094734192, 0.1457452028989792, -0.1439862698316574, -0.06419947743415833, 1.3407586812973022, 1.398245930671692, 1.2222474813461304, 0.933664083480835, 0.7681466937065125, 1.3851238489151, 0.1599322110414505, -1.4643419981002808, 0.9688794612884521, 1.160456895828247, 1.5518052577972412, -0.4774901568889618, 0.13375067710876465, -0.9812260866165161, 0.09723198413848877, -0.6160835027694702, -0.1475548893213272, -0.3493008315563202, 0.40861397981643677, 0.4231407642364502, 0.9731481075286865, -0.4678453803062439, 0.47737932205200195, 1.0539124011993408, 0.999523401260376, 0.7446565628051758, 0.1018187627196312, -0.828522801399231, -1.5413298606872559, -0.41909927129745483, 0.16872136294841766, 2.166369915008545, -1.4202795028686523, -0.6780704855918884, 1.238738775253296, -0.7980579137802124, 0.7149384617805481, 0.5831933617591858, 1.12252676486969, 0.18622015416622162, 0.5415859222412109, -1.488874077796936, 0.48401468992233276, 0.3830718398094177, -0.2744395434856415, 0.7212016582489014, -0.6208011507987976, -1.4637770652770996, -1.833223819732666, -1.189074158668518, 0.7917295098304749, 1.0041940212249756, 0.7737146019935608, 0.46318143606185913, 0.2878040671348572, 0.047656089067459106, -0.8100357055664062, -1.2922210693359375, -1.2252839803695679, -0.28718507289886475, -1.498079538345337, 0.6189830899238586, 1.0922669172286987, -0.23576831817626953, 0.7508271932601929, 0.3462671637535095, -2.0984392166137695, -0.2804044187068939, -1.3529893159866333, -0.03386756032705307, 0.03534933924674988, 0.9998992085456848, -0.8960185647010803, 0.47126153111457825, -1.2273027896881104, -1.5583499670028687, 0.6790717840194702, 0.5688859224319458, -0.7660727500915527, 0.7480105757713318, -1.5577785968780518, 0.5600820183753967, 0.6934095621109009, -0.6507090926170349, 1.1493953466415405, 0.31404149532318115, -0.6899386048316956, -1.1485447883605957, 2.6446621417999268, -1.3796050548553467, -1.4721462726593018, 0.39734914898872375, 2.459660530090332, -1.1991020441055298, -1.0518324375152588, 0.7390730977058411, 1.3310635089874268, 0.3357817828655243, -0.07353508472442627, -1.8152523040771484, 0.5877322554588318, -1.4184046983718872, -0.7517359852790833, 0.12036185711622238, -1.1605219841003418, -0.15053461492061615, -1.023250937461853, -1.7658028602600098, 0.03688056766986847, 0.5330962538719177, 1.4919954538345337, -1.5931473970413208, -0.8135464787483215, 0.07230573147535324, -0.3924402892589569, -0.30177566409111023, 0.8589362502098083, -1.8562564849853516, -0.08275934308767319, -1.7578767538070679, -1.2971134185791016, 1.3550333976745605, 0.8861340284347534, -1.1438227891921997, -1.1736663579940796, 1.2647364139556885, -0.6804341673851013, 0.6547163724899292, -0.504726231098175, 1.6631653308868408, -1.4749858379364014, 1.1488072872161865, 1.2725683450698853, 0.9282490015029907, -1.3586201667785645, -0.9441807270050049, -0.9663088321685791, -0.8845089077949524, -0.8252597451210022, 0.7797380685806274, 1.1662777662277222, 1.9197207689285278, 1.438318133354187, -0.1765386015176773, 0.31427109241485596, 0.20565786957740784, 0.3914792537689209, 2.452418804168701, 0.8588402271270752, -0.7987146377563477, -0.048719849437475204, -0.7262946963310242, -0.18114186823368073, 1.9606050252914429, 0.49964845180511475, -2.3582260608673096, 0.28300443291664124, 0.7811945080757141, -0.19230452179908752, 0.3993763327598572, 1.773350715637207, -0.290873646736145, -0.495327353477478, 0.08214924484491348, 0.21375040709972382, 0.46751466393470764, 2.1224160194396973, 0.9749712347984314, -0.5343546271324158, 1.3029886484146118, -0.9164258241653442, -0.42443427443504333, -1.7702897787094116, 0.2154260128736496, -1.468143343925476, 0.9909725785255432, -0.6265987157821655, 0.034944258630275726, -2.3463613986968994, 0.9061398506164551, -0.5983433127403259, 1.1739466190338135, -1.5430856943130493, -0.33070874214172363, 0.031367670744657516, -1.5570075511932373, 0.21333499252796173, -0.19252248108386993, 0.1936294585466385, 1.416406273841858, -0.511917769908905, -0.9133076071739197, 0.38088932633399963, 0.30439189076423645, 0.7394143342971802, 0.6189853549003601, -1.5297060012817383, -0.20840579271316528, 0.107956163585186, -0.23987631499767303, -0.05003245547413826, -0.5436843633651733, 0.8861722350120544, 0.649474024772644, -0.5759294033050537, -0.7668086886405945, 1.2181954383850098, 2.9545042514801025, -1.2795078754425049, -0.557171106338501, 0.3308272361755371, 0.015248660929501057, 0.8783701658248901, -1.542655110359192, -0.6630730032920837, 1.1496156454086304, 0.27752140164375305, -0.8800718188285828, 1.3968425989151, -1.3195116519927979, -0.34894874691963196, 1.6488584280014038, -1.2027126550674438, -0.11331228911876678, 0.1423780620098114, 0.1660735160112381, -0.18744271993637085, -0.9012269973754883, 0.8742050528526306, -0.8945934176445007, 0.39381712675094604, -0.8158926367759705, -0.2447766363620758, -0.12576358020305634, -0.6590115427970886, -0.8997432589530945, 1.999946117401123, 0.9038304686546326, -0.4056958258152008, -0.7086976170539856, -0.05857374146580696, -0.6428931951522827, 1.2334192991256714, -0.1153506264090538, 0.16750192642211914, 1.9561482667922974, 1.2043737173080444, -1.111470341682434, 1.6703252792358398, 0.9110177159309387, 1.2635005712509155, -1.152057409286499, -0.005803278647363186, 0.7563371062278748, 1.3603111505508423, -3.0016770362854004, 0.5333834290504456, -0.0341399610042572, -1.2498250007629395, -0.07874591648578644, 0.835178017616272, -0.5354606509208679, -0.3272331953048706, 1.3054200410842896, 0.10072610527276993, -0.8838924765586853, -0.3265237808227539, 1.401870608329773, -1.083375096321106, -1.970384955406189, 0.5170099139213562, -1.3494691848754883, -0.6366775035858154, -0.78556227684021, 0.22823983430862427, -0.5785912871360779, -0.2425718605518341, -1.2648426294326782, 0.679705798625946, 0.8090532422065735, 0.565535306930542, -0.7714911699295044, -1.7975550889968872, 0.5252334475517273, -1.9401497840881348, 0.5737646818161011, 0.2759034037590027, 0.2085975557565689, -0.10659986734390259, -0.17696309089660645, -0.9869723320007324, 0.9084085822105408, -0.46205130219459534, -1.345994234085083, 0.2951456606388092, -1.1683522462844849, 0.030615415424108505, -0.356362909078598, -0.3052053153514862, -0.4201262593269348, 0.15581780672073364, 1.640726089477539, 0.3356231451034546, -1.6589561700820923, -1.4562770128250122, -0.2902560830116272, 0.4022783637046814, -0.27704739570617676, -0.17764684557914734, -0.1487181931734085, 0.8394805788993835, 1.345163345336914, 1.958785891532898, -0.5687770843505859, -0.31113043427467346, 1.3758710622787476, -0.6299737095832825, -0.16613931953907013, 0.5198586583137512, 0.447467178106308, -1.4629327058792114, -1.1043049097061157, -1.426857590675354, 1.7977409362792969, 2.2034804821014404, 0.4805662930011749, 0.5983152985572815, 0.1585666835308075, 1.701641321182251, 0.27369052171707153, -0.9571908712387085, -1.2648836374282837, -0.5059114098548889, 0.2704325020313263, -0.6702022552490234, 1.3960442543029785, -0.3342302739620209, 0.8659682869911194, 0.1404767483472824, -0.15688857436180115, -0.8002563714981079, -1.000287652015686, 0.8126897215843201, -0.10444622486829758, -1.396833062171936, -1.1023333072662354, 0.3387860059738159, -1.1457464694976807, -0.9661272764205933, 0.2471105456352234, 1.1402032375335693, 0.04019404947757721, -0.2684009373188019, -0.288379430770874, -0.44921037554740906, 1.0824850797653198, -0.42672914266586304, -0.37761902809143066, -0.34319543838500977, -1.817739725112915, 0.025509798899292946, 0.057569533586502075, 1.2964520454406738, -1.3480185270309448, -0.3574782907962799, 0.34817540645599365, -1.3884693384170532, -0.1170923262834549, 0.555867075920105, -0.25977540016174316, 1.5832597017288208, 0.4092310070991516, -1.5143866539001465, 0.1288658082485199, -0.1660614311695099, 1.3521020412445068, 0.02860790118575096, -0.2196734994649887, 2.125631332397461, 0.5024045705795288, 1.2669057846069336, 0.7628841996192932, -0.16340643167495728, -0.6339953541755676, 1.8497648239135742, -0.22739721834659576, -1.0866128206253052, 1.0936741828918457, -0.2449912279844284, 0.31241366267204285, -0.3705281913280487, 1.065233588218689, -1.13727605342865, 0.030298862606287003, -1.1725189685821533, -0.793703556060791, -0.27544718980789185, -0.7216759324073792, -0.07063273340463638, 1.1960557699203491, 1.3354129791259766, 1.0469574928283691, 0.25280889868736267, -0.4557109773159027, -0.34589093923568726, 0.26336753368377686, 1.4754053354263306, -1.3269459009170532, 2.201430320739746, -2.2633543014526367, -0.9529728889465332, 1.2570003271102905, -0.21849922835826874, 0.6741201281547546, 0.9268637299537659, -0.48206770420074463, -1.0351577997207642, 0.9501195549964905, 0.55766761302948, 0.6945555210113525, -0.2559492886066437, -0.13453564047813416, -0.43262940645217896, 0.8942264914512634, 1.0315885543823242, -2.0831081867218018, 0.00918461475521326, 0.9990438222885132, 0.5160763263702393, -1.538578748703003, 1.1201096773147583, 1.9442567825317383, -0.22690291702747345, -1.1194955110549927, -1.0611720085144043, -0.12606537342071533, 1.303402304649353, -0.014162303879857063, -0.27647218108177185, 2.2410647869110107, -0.13417892158031464, 1.1092772483825684, 0.34951290488243103, 1.4145503044128418, -0.21263228356838226, -1.7477525472640991, 1.1679086685180664, -0.8672195076942444, 1.0405453443527222, -0.15403597056865692, 1.5550552606582642, 0.5799235105514526, 1.0095566511154175, -0.5179041624069214, 1.1655364036560059, 0.8325871229171753, -1.0827879905700684, -0.5923877954483032, 1.343549132347107, -0.6081835627555847, 0.4172605872154236, -0.9505634307861328, 0.7499162554740906, -0.4866039454936981, 0.7658742666244507, -0.0020818912889808416, 0.6501497030258179, 0.4708062410354614, 1.7715181112289429, -1.0385839939117432, 1.6215472221374512, 0.226400226354599, -0.44622743129730225, -0.42503586411476135, -0.5756754875183105, -1.0011544227600098, 0.2495444416999817, 0.058471810072660446, 0.9662753939628601, -0.11184729635715485, 0.09661824256181717, 0.9535448551177979, 0.3258154094219208, -1.253978967666626, 1.6312958002090454, -2.557042121887207, 1.0590139627456665, -0.8371405601501465, -0.9946482181549072, -1.373517394065857, -0.09884179383516312, -0.5841996669769287, -0.43025052547454834, -0.6637479662895203, 0.6352927684783936, 1.0403022766113281, 0.27766647934913635, 0.61300128698349, -0.5358383059501648, 1.0400890111923218, 0.3706854581832886, 0.40381669998168945, 0.9267577528953552, 0.8787248134613037, 0.34713250398635864, -1.1464287042617798, 1.0886672735214233, 0.22760681807994843, 0.5747531652450562, 0.32360953092575073, 0.5771092176437378, -0.06115484982728958, -0.07933233678340912, 0.12685498595237732, 0.4434939920902252, -0.43615761399269104, -1.8706449270248413, 0.7903985977172852, -0.9638067483901978, -0.010910759679973125, 0.2043287456035614, -0.9891311526298523, -1.676504135131836, -0.013719910755753517, 1.4483373165130615, 0.4737902581691742, 0.37607336044311523, -1.7753584384918213, -2.039072275161743, 1.2138248682022095, -0.7094829082489014, -0.7276821732521057, -0.2771715521812439, 0.7860206365585327, 1.3046112060546875, -0.22549836337566376, -0.5210732221603394, -0.5264282822608948, 0.5598993897438049, -1.1849206686019897, -0.6693914532661438, 2.970872402191162, -1.2853431701660156, 0.019618036225438118, -0.8095453977584839, -2.39526104927063, -0.6166484951972961, -0.9246939420700073, -0.009682913310825825, 1.1473685503005981, -0.4355821907520294, -0.4247036576271057, -0.4218940734863281, 0.8129145503044128, 0.37220367789268494, -0.42816537618637085, -1.873976707458496, -0.4079483449459076, 0.7517158389091492, -2.8876848220825195, -0.029643863439559937, 1.1356183290481567, 0.16427484154701233, -0.3382905423641205, 0.2115638703107834, 0.16515088081359863, -0.9860183596611023, 0.7956609129905701, -0.17021024227142334, 0.8719093799591064, 0.048077214509248734, 0.30527615547180176, -1.0060315132141113, 0.2514483332633972, 0.7242436408996582, -1.0691884756088257, -0.30667829513549805, 0.8780262470245361, -0.4354977309703827, -2.1240832805633545, -0.5947797298431396, 0.8421492576599121, 1.8830925226211548, -0.607650876045227, 0.06855140626430511, 0.0923285186290741, 0.5172949433326721, -0.002705345628783107, 0.2717094421386719, -1.054593563079834, 0.3903631567955017, -0.5399958491325378, 1.0549129247665405, 0.8968364596366882, -0.7111304402351379, -0.00934544950723648, -0.5194869637489319, -1.7987582683563232, 0.2053493708372116, 0.777886152267456, 0.1859220713376999, 0.2524203062057495, 0.20709016919136047, -0.05769846960902214, 0.5179282426834106, 1.1377127170562744, 0.28434160351753235, -0.3447752296924591, -0.17850349843502045, 1.3268336057662964, -0.19282041490077972, 0.16986745595932007, 0.3076915442943573, -1.3086885213851929, 0.733425498008728, -0.6639748215675354, -0.14142991602420807, 0.3164772689342499, 1.3258126974105835, 0.08717666566371918, -0.2909894287586212, -0.553942859172821, -0.11122236400842667, 0.006156295537948608, 1.7455832958221436, -1.1349488496780396, -0.23733854293823242, -0.8092079162597656, -1.049683928489685, -1.468962550163269, -0.6534024477005005, 0.5859456658363342, -0.46502023935317993, 0.5874812006950378, 0.7000417709350586, 2.146730899810791, -0.44775858521461487, -0.12938205897808075, 1.6870664358139038, -0.5958504676818848, -1.098421335220337, 2.728771209716797, -0.7712225914001465, 0.31334421038627625, 1.138969898223877, 1.2529308795928955, -0.4677247703075409, -0.8996553421020508, 0.32371172308921814, -1.6002705097198486, -0.225971981883049, -0.8703680038452148, 1.2052042484283447, -0.16961880028247833, -1.709709644317627, 0.086313895881176, -0.9348649978637695, 0.39203837513923645, 0.0433172769844532, 1.229467511177063, -1.2431490421295166, 0.7459799647331238, -0.3550076484680176, 1.5585139989852905, -1.132353663444519, 0.21084892749786377, -0.22832657396793365, 0.7482491731643677, -1.726698398590088, 0.5543324947357178, 0.16587364673614502, 0.6027550101280212, 0.2964477241039276, -0.8305075764656067, 1.735573172569275, 0.622826337814331, -0.7853338718414307, -0.7183206677436829, -1.56038236618042, 0.1807742416858673, 1.4827258586883545], [-0.9437457323074341, 0.20620176196098328, -1.6661423444747925, 1.3092678785324097, -0.2862125337123871, -0.48757919669151306, 1.1981152296066284, -0.23800453543663025, -1.392928957939148, -1.8708847761154175, 0.6274204850196838, 0.20567572116851807, 2.120359182357788, 1.009620189666748, -0.3801678717136383, -1.166154384613037, 0.2713147699832916, -0.4967763125896454, -0.725244402885437, -1.3579870462417603, -0.38514259457588196, 0.021485723555088043, 0.6010326147079468, -1.005399465560913, 0.5806872248649597, 1.2526137828826904, 0.44929906725883484, -0.7844570875167847, -1.1313825845718384, 0.10906989872455597, -0.8642793893814087, -1.7697643041610718, -0.619944155216217, 0.48571380972862244, -0.5929812788963318, -0.8740219473838806, -0.889829695224762, 0.14574317634105682, -0.9973300695419312, 0.04074188694357872, -0.7850110530853271, -1.5770249366760254, -0.6187043190002441, -1.223915934562683, 0.4881444275379181, -0.6028216481208801, -0.07127179950475693, -0.6669108867645264, 0.11328906565904617, 0.11258019506931305, -0.17456667125225067, -0.9728484749794006, 0.1563071757555008, -0.11866980046033859, -2.012166976928711, 0.43801140785217285, 0.06520087271928787, 0.36559510231018066, 0.8419007658958435, 1.2125468254089355, 1.7069014310836792, -0.4840782880783081, -1.0112618207931519, 1.7694592475891113, -0.03055666759610176, -0.11105051636695862, -1.4045346975326538, -1.3303823471069336, 1.2942273616790771, -0.4055143892765045, 0.43861281871795654, -1.2266240119934082, 0.3696451485157013, 0.28902995586395264, 1.3697758913040161, -0.5860444903373718, 0.21407289803028107, 1.1067116260528564, 1.3836992979049683, -0.41107070446014404, -0.5107201337814331, -0.08511694520711899, -0.3896883428096771, -2.4741714000701904, 0.9343863129615784, 0.15423958003520966, -0.1045471578836441, -0.3148423731327057, -0.8704628944396973, -1.2565748691558838, 1.5454260110855103, -1.1913127899169922, 0.8793987035751343, -0.1831851601600647, -0.8078330159187317, -0.04287111759185791, -0.14985936880111694, -0.6906147003173828, -1.3343839645385742, -1.36946439743042, 0.11999475955963135, -0.39986616373062134, -0.7850368618965149, -0.583031177520752, -0.003382063005119562, -0.15185673534870148, 0.14120866358280182, -0.866925835609436, 1.0984162092208862, 0.41335529088974, 0.874698281288147, 1.4030250310897827, 0.9066054224967957, -0.24280239641666412, -0.5522925853729248, 0.6982978582382202, 0.4951198995113373, -0.06019315868616104, -0.011036699637770653, -1.3943274021148682, -0.24350306391716003, -1.3146034479141235, -0.4595099687576294, -1.2678375244140625, 1.505575180053711, -1.0065377950668335, -1.9418983459472656, -0.08880498260259628, 1.2313343286514282, 2.279111385345459, -0.021104874089360237, 1.0493758916854858, 0.28419819474220276, -1.0800706148147583, -0.7958119511604309, -0.6031544804573059, 0.23330888152122498, -1.531983494758606, -0.31967219710350037, -0.9922634959220886, 0.7338809967041016, -1.8840609788894653, 0.8467491269111633, 0.021312035620212555, 1.4842753410339355, 0.311196893453598, 2.4565622806549072, 2.240943193435669, 0.9790545105934143, 1.0648505687713623, 2.128202438354492, 1.3356856107711792, -0.2993440330028534, -0.7260868549346924, 0.08426753431558609, 0.6371991038322449, -0.10208529978990555, -0.4674425721168518, 0.18341004848480225, 0.42399901151657104, 1.0013374090194702, 0.5406169295310974, 0.5032999515533447, 0.4507654309272766, -1.746792197227478, 0.19777549803256989, -0.11853523552417755, -1.7852394580841064, 0.15744364261627197, -0.4950427711009979, 0.5487792491912842, 0.43237367272377014, -0.24256554245948792, 0.02261640690267086, -0.1920439600944519, 0.2751077115535736, -0.1956895887851715, -1.2017544507980347, 1.0344438552856445, 2.035651922225952, -1.2472347021102905, -0.17526879906654358, 0.9909368753433228, 1.1469258069992065, 0.6541954278945923, 0.178376704454422, 1.5298763513565063, -1.1402157545089722, -1.2780214548110962, -2.208186626434326, -1.2123892307281494, 0.7494744062423706, 0.8449910283088684, -1.1329044103622437, 1.685539960861206, -1.1137298345565796, -0.3728001117706299, 0.8427931070327759, -1.5989317893981934, 1.0240304470062256, 0.7421509027481079, 1.550111174583435, 0.7180904746055603, -0.7676084637641907, 0.000257393519859761, 1.3474605083465576, 0.5073226690292358, 0.09713054448366165, 0.519933819770813, 0.831830620765686, -1.6278990507125854, 1.5509169101715088, -0.7091653943061829, -0.5225533843040466, 0.6820254325866699, -1.3011969327926636, 0.5762249231338501, -1.7774242162704468, 1.5514819622039795, 0.5452197790145874, 0.4484069347381592, -0.22550919651985168, 0.1145891323685646, 0.8691161870956421, -0.5314204692840576, 0.6223993301391602, -0.6006665229797363, 1.156313180923462, 0.19201143085956573, 0.707481324672699, 0.46600744128227234, -1.3612244129180908, -0.5961980223655701, 0.542685329914093, -0.2965894639492035, -0.016495581716299057, 0.5740472078323364, 0.46402522921562195, -0.3800336718559265, -0.5052048563957214, 0.2627753019332886, -0.21224038302898407, 0.9459608793258667, 0.44780564308166504, -0.836635172367096, -1.8857663869857788, -1.9453400373458862, -0.1340254247188568, -0.4759323000907898, -0.40268707275390625, 0.7084055542945862, -0.2567436397075653, -0.07342826575040817, 1.5819011926651, -0.45591065287590027, -0.5716837644577026, -0.17406423389911652, -0.6817996501922607, -0.889370322227478, -1.765232801437378, -1.664808988571167, 1.3368052244186401, -0.432989239692688, 1.726385235786438, 0.16081546247005463, -0.07392624765634537, -0.8791548013687134, -0.7382586598396301, 0.9079039096832275, 2.5659611225128174, -0.630952775478363, -0.5995709300041199, -0.38313430547714233, 1.0998388528823853, -1.3435471057891846, 0.96825110912323, 0.8115174174308777, -0.12636949121952057, 0.28463613986968994, -1.7250831127166748, -1.0907827615737915, 0.8476948142051697, 1.854142665863037, 1.4447505474090576, -1.5134270191192627, 0.07983148097991943, -1.6496822834014893, -0.6129558086395264, -0.5347509384155273, -1.614285945892334, -2.1009724140167236, 1.3990159034729004, -0.13493172824382782, -0.31356799602508545, -0.5587974786758423, 1.5490880012512207, -1.0357601642608643, 0.036468926817178726, -1.338618278503418, 1.0305129289627075, 1.378610372543335, 0.4624069631099701, -0.013272387906908989, 0.8720903396606445, 1.5336531400680542, -1.079769492149353, 1.1324853897094727, -0.4849204123020172, 0.31222203373908997, -0.7012284398078918, -0.37679004669189453, -0.3790329098701477, -0.15197397768497467, -0.08346780389547348, 0.6834529638290405, 1.0896153450012207, 0.5480926632881165, -2.171869993209839, -0.32413938641548157, 0.5198268890380859, -0.5250722765922546, 1.03544020652771, 0.930892288684845, 1.0269831418991089, 1.0976645946502686, 0.9765799045562744, -1.03435218334198, -2.035275936126709, -0.3812445104122162, -2.1534783840179443, 3.2592599391937256, -0.3753579556941986, 0.14574936032295227, 0.13255010545253754, -0.45166638493537903, -0.9376055002212524, 2.493411064147949, -1.1229348182678223, -0.10913074761629105, -0.15928645431995392, 0.24984516203403473, -0.32016512751579285, -0.3407108783721924, -0.9225848317146301, 1.1552870273590088, 0.009564626030623913, -0.5550529956817627, 1.024182915687561, -1.894273042678833, -1.7496267557144165, 0.603001058101654, -0.803399920463562, 1.2933160066604614, -0.043395545333623886, -0.1349298506975174, 1.0704830884933472, -0.3510551452636719, -0.081318698823452, 1.1916654109954834, -0.5018623471260071, -2.084449052810669, -0.8573830127716064, -0.5385128855705261, 1.2095766067504883, 1.4762786626815796, -0.8552262187004089, 0.6473708152770996, -1.9593796730041504, 0.4087158143520355, -0.2587299346923828, -0.540070116519928, 0.1389002650976181, -0.41102102398872375, 0.50979083776474, 0.5916361212730408, 0.11758287996053696, 0.18633800745010376, 0.281335711479187, -1.3766237497329712, -0.1622883379459381, -0.0029221775475889444, -0.31230270862579346, -0.26872339844703674, 0.10400829464197159, -0.8355545997619629, 0.16167914867401123, -0.7425934076309204, -0.9102575182914734, 0.11257404834032059, -0.9447277784347534, 0.798353910446167, -1.3518162965774536, 1.9521732330322266, -0.4454343616962433, 1.2030787467956543, 0.18568222224712372, 0.09816045314073563, 1.4098565578460693, -0.2134372889995575, -1.081697702407837, -0.061830371618270874, 0.39161989092826843, 1.1608160734176636, -0.5416406393051147, -1.0528717041015625, -0.9629828929901123, -0.3558940887451172, -0.4798559546470642, 0.5296027660369873, -0.05515315756201744, 0.325570285320282, 2.38065767288208, 0.165864959359169, 2.24517560005188, -0.8323608636856079, -0.9713104963302612, 1.625487208366394, 0.9741138219833374, -0.5597973465919495, -0.14822007715702057, -1.6121108531951904, 0.48805928230285645, -1.9067862033843994, -0.6983673572540283, -2.08353853225708, 0.8651745319366455, -0.5426428914070129, 2.1162633895874023, -0.027220439165830612, -0.6201103329658508, 0.7940834164619446, -0.6135317087173462, 0.20771798491477966, 0.8284714221954346, -0.5741788148880005, -0.5205951929092407, -0.40391838550567627, -0.7220510840415955, -1.1747291088104248, -1.1183944940567017, -0.18395623564720154, 0.8749564290046692, -0.20729991793632507, -1.1612780094146729, -0.7971933484077454, -1.0996047258377075, 0.00578750716522336, -0.2569819390773773, -0.5798658728599548, 0.4879700541496277, -0.48020729422569275, 0.1793680191040039, -1.0874911546707153, 0.19636216759681702, -0.33696290850639343, -1.0421576499938965, 0.8851008415222168, -1.6538242101669312, 1.0113996267318726, 0.11871638894081116, -0.08039695769548416, -0.734358549118042, 0.4946967661380768, 0.20104776322841644, -0.8480050563812256, 2.0144479274749756, 0.6636812090873718, -0.8722851276397705, 1.1400374174118042, 0.09319911152124405, -1.5043489933013916, -0.995657742023468, 1.9938801527023315, -0.47822174429893494, 0.6230416893959045, 0.6691440343856812, 2.0998809337615967, -1.5377150774002075, 0.42295321822166443, 0.6414821743965149, 0.14686119556427002, -1.3888323307037354, -2.196131467819214, -0.16332978010177612, 2.517861843109131, 1.4083092212677002, -0.47067296504974365, -0.8426262140274048, 1.0579781532287598, -0.45417356491088867, -0.7635373473167419, 0.6765040159225464, 0.49513640999794006, 0.9002320766448975, 0.13743804395198822, -0.16613847017288208, 0.7274152040481567, -1.7042313814163208, 0.8290947675704956, -0.6513795256614685, 0.7366029024124146, 0.623568594455719, -1.4083489179611206, 1.2804311513900757, 0.32670655846595764, 0.12200182676315308, 2.101120710372925, 0.1433905065059662, -2.516740083694458, -0.9769434332847595, -0.605923593044281, -0.48685935139656067, -1.4931493997573853, 0.003596313064917922, 0.34095343947410583, 0.2067878246307373, -0.6788198351860046, 0.3967043459415436, 0.022111330181360245, -0.21463121473789215, 0.18605171144008636, -1.4740548133850098, -0.0973910540342331, -0.5175272226333618, -0.5296785831451416, 0.1555662751197815, 2.3344762325286865, -0.5002996325492859, -2.8528082370758057, -0.28815484046936035, 0.18727999925613403, -1.3351435661315918, -0.3456842005252838, -0.5150780081748962, -0.20034515857696533, 1.1537446975708008, 0.0031565811950713396, 0.1799597591161728, -1.182593584060669, -1.873489499092102, 1.3384125232696533, -0.7871113419532776, 3.0776286125183105, 0.44834157824516296, -1.5239222049713135, -0.43382948637008667, -0.9291874766349792, 1.1210768222808838, -0.039251942187547684, -1.564780354499817, 0.19258226454257965, -0.9121060967445374, 0.05862491577863693, -1.6716707944869995, 1.6301926374435425, 0.9757027626037598, -0.5683014392852783, -0.5987250208854675, 1.5319995880126953, -1.189188003540039, 0.7836114168167114, -1.1011348962783813, -1.6811398267745972, 0.4466058015823364, -1.4126468896865845, 0.10163968801498413, -0.0016776076517999172, -1.1809357404708862, -0.19440707564353943, -1.323257565498352, -0.6908369064331055, 0.3806026875972748, -2.8930373191833496, -0.04435384273529053, 0.7190647721290588, 0.20067687332630157, 0.4068337380886078, 1.5609501600265503, 0.19801443815231323, -0.2707684338092804, -0.03255729749798775, 0.10804443806409836, 0.6924837827682495, -0.8955920338630676, -1.4997543096542358, 0.20389655232429504, -0.43237996101379395, 1.6621029376983643, -0.9818984866142273, 1.7851953506469727, 0.821840763092041, -0.08359108865261078, -0.4349142014980316, 0.1299266219139099, 0.5701683759689331, -0.32881227135658264, 0.9902287721633911, 0.5712384581565857, -0.5227207541465759, -1.3056825399398804, 1.8139162063598633, 0.5179099440574646, -0.48487192392349243, 0.44054388999938965, -0.39556071162223816, -1.0447076559066772, -1.3686200380325317, 0.2191506177186966, 0.5284766554832458, -1.8809230327606201, -0.32276299595832825, -0.5014975070953369, -0.8781297206878662, -1.2916136980056763, -0.9496780633926392, -0.789911150932312, 0.18657530844211578, -0.12821219861507416, -1.660771131515503, -0.09935829788446426, 0.546302855014801, -2.5973145961761475, 0.34364134073257446, 0.7652929425239563, -1.1405880451202393, -0.38455161452293396, 1.1952147483825684, 0.7307218313217163, 0.06084711104631424, 1.1965237855911255, 2.007422924041748, 1.2591075897216797, 0.7411171793937683, -0.8606483936309814, 0.5388257503509521, -1.8799712657928467, -0.04977279528975487, -0.46469324827194214, 0.5456419587135315, -1.6657156944274902, -0.3321504592895508, -0.798359751701355, 0.11665786802768707, 1.2316330671310425, -0.9147455096244812, 0.48135286569595337, -1.2772272825241089, -0.3298855721950531, 2.650890827178955, -1.7534791231155396, 0.5577725172042847, 0.6182955503463745, 0.6949787139892578, -1.7738453149795532, -0.39569181203842163, 0.29858288168907166, 0.6022311449050903, -0.7776417136192322, -0.3879533112049103, 0.41472160816192627, 1.3200033903121948, 0.8753136992454529, -0.7825853824615479, -1.7700622081756592, 0.7250165343284607, -0.7917068600654602, -1.1172844171524048, 1.3922533988952637, 1.0655614137649536, 0.06049701198935509, 0.7571788430213928, 1.6092123985290527, -0.594306230545044, -0.6704331636428833, -1.2379374504089355, 0.09754833579063416, -0.9754577875137329, 1.185672402381897, 0.4884280562400818, -0.705740213394165, -0.6636755466461182, 0.6002339720726013, -0.16593793034553528, 0.35716670751571655, 0.42254334688186646, 2.0976409912109375, 0.6053385734558105, 0.11993756890296936, 0.8738823533058167, -0.230118989944458, 1.842889428138733, 0.43839237093925476, -1.2606974840164185, 1.3381656408309937, 1.1046993732452393, 0.51182621717453, 0.09842744469642639, -0.47840186953544617, -1.5560473203659058, -0.1069924607872963, -0.5442303419113159, -0.659142255783081, -0.006360563449561596, -0.6310011148452759, 0.3023046851158142, 0.8200821280479431, 0.7029996514320374, 0.7119649052619934, 0.5958543419837952, -1.311452865600586, -0.007508574984967709, 0.8095185160636902, 0.40341800451278687, -2.4030447006225586, -0.48022082448005676, -1.5200945138931274, -1.6323521137237549, -0.7757850289344788, -0.8061885833740234, -0.05657601356506348, -0.07587996125221252, -0.20124290883541107, -1.5684574842453003, 1.3265156745910645, 2.0549099445343018, 0.11469035595655441, 1.4377374649047852, -0.6772701740264893, 0.36121806502342224, -0.16028456389904022, -0.4747079312801361, -0.23277509212493896, 0.5584608912467957, -0.8498126268386841, 1.146759033203125, -0.34021544456481934, 1.076980710029602, -0.9301439523696899, 0.4855656027793884, -0.992330014705658, 0.10194627195596695, -0.16129644215106964, -0.3382214307785034, 1.1208337545394897, 0.6730294823646545, 1.9689630270004272, -0.7116473913192749, -1.6344513893127441, 0.11137555539608002, -0.9088165163993835, -0.24089936912059784, -0.07365275919437408, -0.5148142576217651, -1.2136021852493286, 1.206986665725708, -0.3158870339393616, -0.9502660632133484, 0.4127703607082367, -0.3651030361652374, -0.32769128680229187, 2.4473178386688232, 0.4414956569671631, 0.5491640567779541, 0.20412947237491608, -0.37199318408966064, -0.1917654573917389, 0.10269410163164139, 1.2207317352294922, -0.1363542228937149, 1.839653730392456, -0.3395148515701294, -1.4805055856704712, 0.35985273122787476, 0.850632905960083, 1.7636739015579224, -1.268386721611023, 2.524240255355835, -0.3554431200027466, -1.5440460443496704, -0.008583052083849907, 0.027755605056881905, 0.5993573665618896, 2.3051023483276367, -0.8882066607475281, -0.5297570824623108, 1.2619507312774658, -1.1562700271606445, 0.5824723839759827, 1.202834963798523, -0.5244936943054199, -0.40462958812713623, -0.5400658845901489, 1.5719465017318726, -0.08628657460212708, 0.006864174734801054, 0.19296211004257202, -0.2270464450120926, 0.8891444206237793, -0.3679826557636261, -1.2245022058486938, -0.777910053730011, -0.7244516015052795, 0.10832051932811737, -0.15051905810832977, -0.15501506626605988, 0.05057581886649132, -0.7533116936683655, 1.076062560081482, -0.6175958514213562, 0.7219235301017761, 0.014956311322748661, -0.207872211933136, -0.34319838881492615, 0.617548942565918, 0.2249620407819748, -0.42073336243629456, 1.368219017982483, -0.10425868630409241, -1.660120964050293, -2.0682098865509033, -0.6075056791305542, -0.22867350280284882, 0.08875396102666855, -0.2768457233905792, -0.09411568194627762, -1.0695409774780273, 0.2587928771972656, -0.5182442665100098, -0.23468181490898132, -0.029083944857120514, 0.5612912774085999, 0.5716800093650818, 0.18629296123981476, -0.6221933364868164, -0.2267688363790512, -1.4112051725387573, 0.6252473592758179, 0.44488200545310974, -0.5540961027145386, -0.2980194687843323, -2.1456551551818848, -0.6461127400398254, -1.498039722442627, 0.21510130167007446, -0.9118356108665466, 1.7837467193603516, -1.0720621347427368, -1.0333229303359985, -0.7729431986808777, -1.2221248149871826, 0.8804353475570679, -0.35956037044525146, -0.9659709334373474, 2.630589008331299, 0.13714934885501862, -0.006506727077066898, 0.6445454359054565, -0.6912513375282288, -0.3467126488685608, -1.4084832668304443, -0.7782372236251831, 0.2596851885318756, 0.22598019242286682, 1.7024427652359009, 0.09106519818305969, -1.8809208869934082, -0.10434860736131668, 1.7274830341339111, 0.05872432515025139, 0.6055021286010742, 0.7368717193603516, -0.15912295877933502, -0.09293022006750107, -0.3203819692134857, -0.40257391333580017, 0.13927264511585236, 0.3007696270942688, 1.4722071886062622, 2.9298903942108154, -0.4600491225719452, 0.7178208231925964, 0.08268781751394272, 0.8269075155258179, -1.1662131547927856, 1.108795404434204, 1.2711390256881714, 0.2367287129163742, 0.900467574596405, -0.523124098777771, -0.2823035418987274, 0.4489147365093231, -1.369413137435913, -0.6501404643058777, -0.1719297468662262, 0.3744065463542938, 0.07796405255794525, -1.6956955194473267, -0.4302641749382019, 1.4581851959228516, 0.017640134319663048, 0.597593367099762, -0.8495045304298401, 0.6259239912033081, -1.8130980730056763, -0.33724457025527954, -0.5510170459747314, 1.539228916168213, 0.41156989336013794, 0.905796229839325, -0.19143536686897278, -0.33649009466171265, 0.11172819137573242, -1.182000994682312, -0.584183931350708, -0.3811533749103546, 1.0569156408309937, -0.09880656749010086, -1.026916742324829, 0.570122241973877, 0.8513811826705933, -1.183261752128601, 0.04869154840707779, 0.84513920545578, 0.3179285526275635, 0.31694719195365906, -0.6901861429214478, 1.4762412309646606, 0.22733385860919952, -0.5307928919792175, 1.1166296005249023, 0.9345998167991638, -0.4820934236049652, -0.413027286529541, -0.1571548581123352, 0.6375848054885864, 0.6652888059616089, -0.08707550913095474, 1.3719806671142578, -0.7642202973365784, -0.8284077644348145, 1.5628191232681274, -0.4724298417568207, -0.4380001127719879, -0.9389419555664062, -0.012632640078663826, 0.2962071895599365, 2.087092399597168, 0.2214442640542984, 0.945884644985199, 0.9663180708885193, 0.3223835825920105, -0.24284790456295013, 2.4919474124908447, -2.2750988006591797, 0.1219443753361702, 1.8874528408050537, -0.984771192073822, 0.2006378173828125, -0.3209623694419861, -0.38576656579971313, 1.3253397941589355, -0.481404572725296, 0.21103911101818085, -1.2464390993118286, -0.7690194845199585, 0.6022266745567322, 0.8496025800704956, -0.8348580002784729, 0.614606499671936, -0.9563097357749939, 1.8088738918304443, 0.9728096723556519, -1.0297049283981323, -1.40212881565094, 1.0759265422821045, -0.24260757863521576, -0.7382092475891113, 1.331566572189331, -0.015204044990241528, 2.432480573654175, -1.234047293663025, 2.8755130767822266, 0.04961652681231499, 0.9034645557403564, -1.229413390159607, -0.06021815165877342, 0.6397838592529297, 0.06971494108438492, -0.2218971997499466, -1.5975425243377686, -1.277985692024231, 0.9296243786811829, 0.4237871766090393, 0.8641146421432495, 2.507878541946411, 0.0725562795996666, -0.0804048553109169, -0.248366117477417, -0.8490350246429443, -0.03232993185520172, 0.0054323491640388966, 1.1463674306869507, 1.8560620546340942, -0.2929077446460724, 0.9381200075149536, -0.5798275470733643, -0.3424290418624878, -0.19119898974895477, 0.6252661943435669, -1.4815268516540527, -0.4861205518245697, 0.9834937453269958, -0.44486042857170105, 1.3126215934753418, -0.3606029152870178, 0.908700168132782, 0.5087807774543762, -1.546397089958191, -1.2548635005950928, 1.6676483154296875, 0.9243453145027161, 1.4183835983276367, 1.7931071519851685, -1.1812282800674438], [0.640023946762085, 0.49445226788520813, 0.7205071449279785, -0.9201603531837463, -0.33930504322052, -0.5284066796302795, -1.160525918006897, 0.1844107210636139, -0.16457785665988922, 0.7837070226669312, -0.8904567360877991, 1.4152872562408447, -0.6116395592689514, 0.13728021085262299, -0.25647735595703125, -0.10505039989948273, -1.2552940845489502, 0.005018472671508789, -1.1531803607940674, 0.30532389879226685, -1.598251223564148, 0.5018201470375061, 0.16384969651699066, -1.616477131843567, 0.29941585659980774, -0.35452765226364136, -0.7494985461235046, 0.6418623924255371, 2.0198798179626465, -0.028872324153780937, -0.8754293918609619, -0.6479253768920898, 1.117187738418579, -0.8304867744445801, 1.4299994707107544, 2.672633409500122, -0.5053689479827881, -0.6024264097213745, 0.3972809910774231, 0.26390835642814636, 1.00570547580719, 1.2022302150726318, 0.29187121987342834, -0.7906919717788696, -0.23345334827899933, 0.43707218766212463, 0.9164608716964722, -0.841184675693512, -1.1924138069152832, -0.35926327109336853, 0.6838839054107666, 1.5973557233810425, -0.11788003146648407, 0.17358030378818512, -0.3077404499053955, -0.5087381601333618, 1.0445489883422852, 0.8960461020469666, 1.3470232486724854, -1.3794831037521362, -0.46653902530670166, -0.4770675599575043, 1.145086407661438, -1.198103904724121, -0.7252263426780701, 0.48332732915878296, -1.1639400720596313, -0.8272544145584106, -0.9462617039680481, -0.5520263910293579, -0.47652214765548706, -1.1481455564498901, -2.0298333168029785, 0.36359503865242004, -2.3594117164611816, -0.7321527004241943, 1.026620626449585, 0.773768961429596, 0.08947288990020752, 0.8466650247573853, 0.45829910039901733, 0.4490380883216858, -0.6245107650756836, 0.28333550691604614, -0.8464368581771851, -0.7720322608947754, -1.5751667022705078, 1.4453160762786865, -1.8119070529937744, 2.3591787815093994, 0.04002033546566963, 0.8812326192855835, -0.21277089416980743, 1.252143383026123, -0.5009959936141968, 0.18665845692157745, -0.5697595477104187, 1.1791356801986694, -1.10697603225708, 0.826992928981781, -1.014365315437317, -0.31456422805786133, 2.0260658264160156, 0.03320429101586342, -1.5727585554122925, 0.9549338817596436, 1.4068610668182373, 0.8321287631988525, -0.6185234785079956, 0.7130656838417053, 0.11935258656740189, -1.2565288543701172, -0.45178091526031494, 0.9426373243331909, 0.7115277647972107, 0.40690845251083374, 0.0709446519613266, -0.15012210607528687, -0.27344033122062683, 0.27765756845474243, 0.5691865682601929, 0.625910222530365, -0.12413971871137619, 0.16049616038799286, 0.9868854284286499, -0.3091251254081726, -0.20911632478237152, -0.9407240152359009, 0.3361908495426178, -0.9991479516029358, -0.08973698318004608, 0.28183406591415405, -1.8570270538330078, 0.6283948421478271, 1.3214068412780762, -0.6760358214378357, -0.006767362356185913, -1.0832207202911377, 0.9317665696144104, -0.09223220497369766, 0.40346306562423706, 0.4116351008415222, -0.785058319568634, 0.07301506400108337, 1.2359329462051392, 0.09107086062431335, 0.9653509259223938, -1.090901255607605, -1.047497034072876, -0.9489399194717407, 2.45038104057312, 0.8866493105888367, -0.21762311458587646, -1.9700039625167847, -0.8364436030387878, -1.6057708263397217, 0.4336402714252472, -0.1913505494594574, 0.3701702952384949, -1.462842583656311, -0.6126508116722107, -0.46435025334358215, 0.15218091011047363, -1.6131291389465332, -0.49528488516807556, 0.06525038927793503, -2.0129246711730957, -0.12513482570648193, -1.2029424905776978, 0.8899460434913635, 0.44759324193000793, -0.1175711527466774, 1.836210012435913, 2.1083881855010986, 0.9714818596839905, -1.0868169069290161, -0.45162233710289, -1.4250844717025757, 0.5292685031890869, 0.32131969928741455, 0.1623300164937973, -0.16097083687782288, -0.0820903480052948, 0.6966317892074585, 0.054763901978731155, -0.05436360090970993, 2.1643788814544678, 0.8516374826431274, 1.037476658821106, 1.920831322669983, -3.1872470378875732, -1.0918314456939697, 1.0532681941986084, 0.8265791535377502, 0.40964338183403015, 0.1687583476305008, 1.5522472858428955, -2.0654709339141846, 0.038310740143060684, 1.6722416877746582, 2.160757303237915, -0.3167977035045624, 0.35415101051330566, 0.32045605778694153, -0.17094990611076355, 0.29657629132270813, 0.38380780816078186, 1.0246539115905762, -0.7377336621284485, -0.11494505405426025, 1.4162899255752563, 0.9776130318641663, -0.3419894576072693, -0.6954513192176819, -0.9203823208808899, -0.9414576292037964, -0.23211008310317993, -1.0422877073287964, -0.7027069926261902, -1.9398270845413208, 0.948143482208252, 1.112965703010559, -1.4622306823730469, 0.22146296501159668, 0.4132338762283325, 0.007581237703561783, -1.5624347925186157, -0.9485600590705872, -1.020551085472107, 2.042276620864868, 1.5595817565917969, -1.0094527006149292, -0.2047598958015442, 1.662457823753357, -0.35329222679138184, 0.3101321756839752, -0.43825292587280273, 0.4858047366142273, 1.0700923204421997, -0.16537421941757202, -1.3695071935653687, -0.4141516387462616, 1.0037095546722412, -0.7090634703636169, 0.5022770166397095, -1.8335376977920532, 1.5170344114303589, 0.274394690990448, -1.7802700996398926, 0.14650891721248627, 0.0723411813378334, -0.292720764875412, 0.5292477011680603, 1.5065767765045166, 0.6288096308708191, 0.0599345862865448, 0.900399386882782, -0.8240653872489929, -1.6776257753372192, 0.28073203563690186, 0.4738926887512207, -0.6398840546607971, 0.7094553112983704, 0.038672275841236115, -1.6117403507232666, 0.21700794994831085, 0.29100483655929565, 2.287747859954834, 0.09724034368991852, -1.2286711931228638, -0.7444285750389099, -0.9379357099533081, 1.1934694051742554, 0.4140377342700958, -1.270848035812378, -0.10776971280574799, -1.314266562461853, -0.28528422117233276, -0.7079879641532898, -0.2705259919166565, 0.9191792011260986, 0.8803048729896545, 0.35429689288139343, -0.27520081400871277, -0.6516395211219788, 0.19356590509414673, -1.219744324684143, -0.8051627278327942, 0.40255796909332275, -0.4089396595954895, -1.09981369972229, -0.38958436250686646, -0.1060558557510376, 0.5066683888435364, 1.401544451713562, 0.2872775197029114, -0.4649670422077179, 1.3667844533920288, 0.7010155320167542, 1.5426208972930908, 0.32047972083091736, 0.6668772101402283, -0.16928409039974213, 0.6290314793586731, 1.0025911331176758, 0.8936017155647278, -1.281113862991333, -0.5372816324234009, 0.5277341604232788, 0.29317447543144226, -1.2966681718826294, -1.1081348657608032, 0.5387091636657715, 0.7729933857917786, 0.6996156573295593, 1.3878625631332397, 1.2733430862426758, -0.5374351143836975, -0.5471948981285095, -2.0405313968658447, -0.042871974408626556, -0.056329395622015, 0.5571798086166382, -0.7060297131538391, 1.6872637271881104, 1.1293907165527344, -0.29090696573257446, 0.17683494091033936, 1.2732609510421753, -1.1393210887908936, 0.5219506621360779, -0.13021941483020782, -1.3056446313858032, -0.5333492159843445, 0.07653343677520752, 0.0399755984544754, -0.3851461112499237, -0.5850669741630554, 1.2228035926818848, 0.5137647986412048, -0.9001263976097107, 1.085447072982788, -0.2748391330242157, 0.18115665018558502, 0.06574845314025879, -0.7637629508972168, 0.04711579158902168, -1.4670608043670654, 0.8499941825866699, -0.4176713824272156, -0.25434133410453796, 1.1648752689361572, -0.7601418495178223, 1.4240446090698242, -0.5541757345199585, 0.14890803396701813, 0.5413292050361633, -1.757300853729248, 0.84039306640625, -0.3326912224292755, -2.7952208518981934, 1.8437544107437134, -0.11780082434415817, 0.4141688644886017, -0.6484026312828064, -1.1254100799560547, 0.49467384815216064, 0.7781624794006348, 1.513395071029663, 0.5342651009559631, 0.746618926525116, 0.23734861612319946, 1.787623643875122, 0.08639172464609146, 0.5303871035575867, -0.3244568407535553, -0.4246891140937805, 1.6104834079742432, 0.07208725810050964, -0.03798188641667366, 1.1492118835449219, -0.8629409670829773, -0.3116510510444641, -1.3932756185531616, 0.474133163690567, -0.8893517255783081, 0.22566474974155426, 1.392118215560913, 1.2981218099594116, 1.5945672988891602, 1.0713478326797485, 0.15271678566932678, 0.19685086607933044, 0.78961580991745, 0.588916540145874, -0.21470238268375397, 1.0654597282409668, -1.7303889989852905, -2.589301586151123, 1.6458332538604736, 0.8122819662094116, -2.8629467487335205, 1.0957633256912231, -0.5248432755470276, -0.7466411590576172, 1.0335139036178589, -0.6867122650146484, 1.6349620819091797, 1.3640947341918945, 0.045864712446928024, -1.2178515195846558, -0.6836665868759155, -0.004195655696094036, -0.6002234816551208, 0.14201129972934723, -0.41143137216567993, 0.03973016142845154, 0.4133554995059967, 2.353576421737671, 0.9465216398239136, 0.45632505416870117, -0.024371353909373283, -1.412690281867981, -1.4040127992630005, -1.4109034538269043, -2.5141518115997314, -1.8185111284255981, 2.373447895050049, -1.115125060081482, -0.7321938276290894, -1.5382262468338013, 0.4770294427871704, -0.5881949067115784, -1.392258644104004, 0.46751198172569275, -0.30774953961372375, -0.12619318068027496, -2.4916763305664062, -0.34517908096313477, 0.6079612970352173, 0.7271257042884827, 0.5824101567268372, -0.730800986289978, 0.8014973402023315, -0.6975225806236267, -1.3052641153335571, -1.4413968324661255, -0.09627535939216614, -1.1737172603607178, 1.7204225063323975, 1.1917805671691895, 0.03364395722746849, -0.7872005701065063, 0.6623557209968567, -0.6167492270469666, 1.5822930335998535, -1.8641761541366577, -1.5907447338104248, 0.7023931741714478, -0.15326033532619476, -0.4005260467529297, -1.5200444459915161, 0.12880004942417145, -1.7987234592437744, 0.8246229290962219, 0.12619741261005402, -0.22100479900836945, -2.0705454349517822, 0.6511735916137695, 0.6711740493774414, -1.632179617881775, 1.0176076889038086, 2.3767237663269043, -0.6567115783691406, -2.0144217014312744, 1.109947919845581, 0.688473105430603, 0.1463112235069275, 1.33171546459198, 1.0698914527893066, 0.37821903824806213, -0.2530863881111145, 0.0246285367757082, -0.24600087106227875, -0.47157543897628784, -0.3838876187801361, -0.2924959063529968, -1.4983422756195068, 1.9407273530960083, 0.6114621758460999, 0.30773237347602844, -0.19714343547821045, 0.2472456991672516, 0.406874418258667, -0.203534796833992, 2.522775888442993, 0.2336379438638687, -1.5520423650741577, -1.1434969902038574, -2.0337517261505127, -0.47557684779167175, 1.2239106893539429, -0.6411840319633484, 0.025586457923054695, -0.8688571453094482, 1.106283187866211, 1.7701973915100098, -0.5267150402069092, -0.19603002071380615, -0.9364107847213745, 0.026607152074575424, 0.08300095051527023, -0.22552227973937988, -2.191765069961548, 0.6657654643058777, -0.4707367420196533, 0.8902674317359924, 1.2230223417282104, 1.7585115432739258, 1.1172987222671509, 0.002488690661266446, 0.30857977271080017, -0.7772469520568848, -0.7409279942512512, -1.469501256942749, 1.0113226175308228, 0.9406818151473999, -0.9770870804786682, 0.011089532636106014, -0.8277194499969482, 1.4861605167388916, -0.3637705147266388, 1.6657562255859375, -0.2806667387485504, -0.5886501669883728, 0.10457195341587067, 0.7959437966346741, 1.5091396570205688, -0.8586096167564392, 0.8369823694229126, 0.5723260045051575, 0.5695754289627075, 2.2598609924316406, -1.6650712490081787, 1.2866265773773193, -1.4726229906082153, -0.3909600079059601, -0.6257554888725281, 0.3902254104614258, 1.454755425453186, 0.1857113093137741, -0.5758183598518372, -1.5216389894485474, -1.1052805185317993, -0.7137932777404785, 0.9368583559989929, -0.4905773997306824, -1.2568131685256958, -1.299664855003357, -0.44958916306495667, -0.24773189425468445, -0.131048783659935, -0.009373367764055729, -0.12450597435235977, -0.6808661222457886, 0.7034510970115662, -1.1847563982009888, -1.3738610744476318, 0.40955615043640137, -0.16916392743587494, 1.2354925870895386, 0.31060275435447693, -0.295621395111084, -0.2589946985244751, 1.106531023979187, -1.3581831455230713, -1.1444616317749023, -1.2467535734176636, -0.8160192370414734, 0.12419979274272919, -1.606541633605957, -0.12497324496507645, -1.0608222484588623, -0.28784337639808655, 0.3421710729598999, -1.5221635103225708, -0.30643466114997864, 0.1494346410036087, 1.0308887958526611, 0.08465882390737534, -1.3062623739242554, -0.020195094868540764, 0.20326925814151764, 2.243492841720581, 0.28401264548301697, 2.719442129135132, 0.7283554077148438, -1.8225455284118652, -0.19859376549720764, -0.6173912286758423, 0.4915449321269989, -0.9020599126815796, -0.8625690340995789, 1.4135639667510986, -1.060146450996399, -0.11345326900482178, -0.2472318857908249, 0.19315055012702942, 2.0326380729675293, 0.2621457874774933, -1.8365896940231323, -0.652472734451294, -0.4508976936340332, -1.2545819282531738, 1.4647589921951294, 0.6808168888092041, 0.20594333112239838, -1.9093742370605469, -1.1829732656478882, -1.3349794149398804, 1.9701000452041626, 1.20637047290802, 0.07251883298158646, 0.14235959947109222, 1.2923214435577393, -0.7758649587631226, 0.6194905638694763, -0.22002564370632172, -0.12296003848314285, 0.2127452790737152, -0.6869986057281494, -0.08224252611398697, 0.2829381227493286, 0.9192020297050476, -0.24530158936977386, -0.7442808151245117, 1.0748307704925537, 2.802887439727783, 0.9673365950584412, 0.25346946716308594, -0.2193971574306488, 0.08831270784139633, 1.6339892148971558, -0.6064009070396423, -0.5489099621772766, 0.7487422823905945, -1.5961930751800537, 0.9078642129898071, 0.5951818227767944, -0.23117519915103912, 0.09296060353517532, -0.47808679938316345, -0.2846637964248657, 0.5180288553237915, -0.46976202726364136, 1.5880649089813232, 0.20680956542491913, 0.5962943434715271, 0.0507088378071785, 0.26204434037208557, -1.4797207117080688, 1.5100843906402588, 0.4914312958717346, 1.1917158365249634, 0.11842431128025055, -0.02748946100473404, -0.4284599721431732, 0.7340537905693054, 0.25577017664909363, -1.240817904472351, -0.8795783519744873, -0.687806248664856, -0.3859840929508209, -0.23231269419193268, -0.7392653226852417, -0.43088093400001526, -0.03367304056882858, 0.3319588601589203, -0.27485191822052, 1.258445143699646, 0.6473604440689087, 1.7665921449661255, -0.07718236744403839, -0.38357529044151306, 0.6807235479354858, 1.7743995189666748, 1.5177724361419678, -0.6797679662704468, -1.8470754623413086, 0.16434486210346222, -0.1527988761663437, -2.44576358795166, 0.8229255676269531, 1.124474048614502, 0.24708980321884155, -1.3854821920394897, -0.6603357791900635, -1.0372895002365112, -0.09162984788417816, 0.09830841422080994, -0.11003854870796204, -0.19485551118850708, 0.31927385926246643, 0.25946417450904846, -0.9084519743919373, 0.7980066537857056, -0.5721566677093506, 1.3167551755905151, -1.097916841506958, 0.07166661322116852, -1.2672454118728638, -0.9637872576713562, -0.8191669583320618, 1.6789751052856445, -0.5654165148735046, -0.28066760301589966, -0.2418287992477417, 0.6907587051391602, -0.802018940448761, -1.1637314558029175, -1.0802375078201294, 0.8916277885437012, -1.2417107820510864, 0.19602273404598236, -0.4685279130935669, 0.26348623633384705, -0.2978988289833069, -0.24108485877513885, 1.1046817302703857, -1.0407147407531738, -0.3781595528125763, 0.27900755405426025, -0.5644538402557373, 0.5254260897636414, -1.5529831647872925, 1.6014060974121094, -0.25267118215560913, -1.4252341985702515, 0.08194749802350998, 0.38047051429748535, 1.3198950290679932, 0.9969151020050049, -0.38047361373901367, 1.1999444961547852, 0.7083899974822998, -0.5252276062965393, 1.1000490188598633, 0.9015427231788635, 1.9537776708602905, -0.6979336738586426, -1.9103796482086182, 0.9464037418365479, 0.9127286076545715, 0.6983328461647034, 0.3871956467628479, -0.6773502230644226, 0.8469752073287964, 0.7275654077529907, 1.2310606241226196, 0.5695263743400574, -1.2474534511566162, 0.9380002021789551, 2.4317266941070557, -2.659762144088745, -0.03486575931310654, -0.6538593173027039, -0.325145959854126, -0.6126342415809631, -2.116456985473633, 1.060734510421753, -0.5295043587684631, 2.268343687057495, 0.0005359236965887249, 0.12658341228961945, 1.0428156852722168, 0.3475581109523773, -0.14409738779067993, 1.4480280876159668, -0.7453540563583374, 0.2796427607536316, -0.5287048816680908, -0.583187997341156, -0.5683791041374207, -0.3920775353908539, -0.5656036734580994, -0.4208838939666748, 0.8141829371452332, 1.3432420492172241, -1.6060439348220825, -1.262146234512329, -1.9130008220672607, 0.04389698803424835, -0.7257996797561646, 0.45559927821159363, -0.13893982768058777, 0.35118281841278076, 0.7687718868255615, 0.8512174487113953, 0.10568798333406448, -0.47213760018348694, 0.49682870507240295, 0.30826741456985474, -0.0018481004517525434, -0.3600243031978607, -1.9324158430099487, 0.3578494191169739, -0.768174946308136, -0.9405030608177185, 0.3311880826950073, 0.16615402698516846, 0.15027178823947906, -0.2529394030570984, -1.5080573558807373, 0.5649195909500122, -1.5017539262771606, -0.41084030270576477, 0.6703991293907166, 0.7053580284118652, 0.031467095017433167, -0.6760931611061096, 1.4343124628067017, -0.5312207341194153, -0.39158400893211365, 0.05302971974015236, -0.016711991280317307, 0.016936300322413445, -0.16185373067855835, -1.2749567031860352, -0.48831236362457275, -0.1977858692407608, -0.22018876671791077, -0.4550416171550751, -0.7849724292755127, -0.38512474298477173, -1.8744038343429565, -1.196463942527771, -0.5406131148338318, 0.030149366706609726, 0.05343053489923477, -1.0614173412322998, 2.031464099884033, -0.46137696504592896, -0.04158773273229599, 0.9537568092346191, -0.7025285363197327, 1.0142364501953125, -1.6468998193740845, -1.6621177196502686, -1.490714192390442, -0.36858251690864563, -1.6672950983047485, -0.955432116985321, -1.457848310470581, -1.4538064002990723, 0.09822146594524384, -0.5351090431213379, 1.0572746992111206, -1.1457489728927612, 0.3635578155517578, 0.8754607439041138, -0.6400167942047119, 0.2152743637561798, 1.2054331302642822, 0.5649361610412598, 1.2054510116577148, -0.7712278366088867, 0.09026850759983063, -1.0830440521240234, -0.7045058608055115, 0.4552679657936096, 0.7424527406692505, 1.1986900568008423, -1.4716174602508545, -0.4029609262943268, -0.7309735417366028, 1.1773570775985718, -0.17395153641700745, 0.819847822189331, -0.6620811223983765, 0.9417468309402466, -1.298887848854065, -0.9785807132720947, 0.4077390432357788, -0.1459406465291977, 0.3961118161678314, -0.6474391222000122, 1.67074716091156, -0.34013473987579346, 0.47693008184432983, -0.6847847700119019, -0.04150167107582092, -0.4684779644012451, 1.2909241914749146, 0.5514642596244812, 0.34465980529785156, 0.7038506269454956, -0.9356269836425781, 0.12875407934188843, -2.5780181884765625, 1.0149013996124268, -0.2930790185928345, 1.4809319972991943, -0.7134013175964355, -1.0610065460205078, 0.918370246887207, 0.40596845746040344, 0.2809118926525116, 0.09565896540880203, -0.44191548228263855, -0.11934500932693481, 0.6345130801200867, 0.4651533365249634, -0.026542972773313522, 0.19134651124477386, -0.19018372893333435, -0.2747572958469391, -1.1720077991485596, -0.10200420022010803, 1.7163498401641846, 0.1920403093099594, -0.789685845375061, -0.7989945411682129, 0.31315577030181885, -1.3765418529510498, -0.7087342739105225, -1.8107903003692627, -0.02684619277715683, -0.005354410968720913, 0.06436123698949814, 0.4370757043361664, 0.5465108752250671, 0.8687283396720886, -0.7687655091285706, -0.08473362773656845, -1.333367943763733, 1.258449673652649, -0.8317116498947144, -0.7756267189979553, 1.1334476470947266, -0.46919065713882446, -0.4807250201702118, 0.7768497467041016, 1.219953179359436, 2.005455732345581, -0.2999829053878784, -0.37909436225891113, 0.5863401889801025, -0.8286795020103455, 1.2812649011611938, -0.00403213407844305, 1.4108649492263794, -0.9982327222824097, -0.6286196112632751, 0.4856937825679779, 0.2024264633655548, 0.12991175055503845, -0.3026455342769623, -0.7238999009132385, 0.27818185091018677, -2.1613426208496094, -0.8026494979858398, -0.5372595191001892, -0.2539393901824951, 0.07945192605257034, -0.10348045825958252, 0.13980883359909058, -0.7423508167266846, 0.5761405229568481, 0.2984466850757599, -0.5744791626930237, -1.0288246870040894, -0.23642829060554504, -0.1719060093164444, -0.27508702874183655, 0.7065523266792297, 0.11593693494796753, 1.2512067556381226, 1.510201334953308, 1.3709776401519775, -0.25436267256736755, 0.5707265138626099, 0.9162925481796265, 0.40373414754867554, -1.305012822151184, 3.3679163455963135, -0.3624732494354248, -0.23477426171302795, 0.671862006187439, -0.5786312222480774, 0.7816523313522339, -0.15699508786201477, -1.2027889490127563, -0.4929563105106354, 0.12515467405319214, 0.7392839789390564, -0.9608150124549866, -0.8525487780570984, 0.5683742165565491, 0.8430812954902649, -0.6922826170921326, 0.45042985677719116, -0.1219881996512413, 0.6961590647697449, -0.4849849343299866, 0.05975443124771118, -1.1281464099884033, 0.1351214349269867, -1.3657712936401367, 0.6563636064529419, 1.2610480785369873, 0.7581367492675781, 0.9462797045707703, 0.42754337191581726, -0.6887174248695374, 1.6682584285736084, 1.0082764625549316, -0.9387389421463013, 0.7025420069694519, 0.32661521434783936, 0.44947993755340576, -0.5470742583274841, 0.43997707962989807, 1.3811115026474, 0.4746536314487457], [1.3191173076629639, -0.17962175607681274, 0.9004892110824585, -0.41407448053359985, -0.9404012560844421, 0.6315049529075623, 0.2476579248905182, 0.16130219399929047, 0.3176119327545166, -0.05738447979092598, 0.21994732320308685, -1.357201337814331, -1.380976915359497, 0.3666499853134155, -1.084177017211914, -1.1723541021347046, -0.8786506652832031, -1.0248826742172241, 1.957535982131958, -0.5242453813552856, -0.8188003897666931, 0.9259613156318665, -1.937976360321045, 0.7325567603111267, -0.1672137826681137, -0.7361172437667847, -0.7354280948638916, -0.3168564736843109, 1.8193541765213013, -0.023053353652358055, -0.9763117432594299, -0.021402111276984215, -1.610870122909546, 0.08225117623806, -1.7750214338302612, -0.5560455918312073, 0.00037780916318297386, 0.641477108001709, 1.0438727140426636, 1.4266148805618286, 1.1694693565368652, 0.597930371761322, -2.021242141723633, 0.2402285933494568, 1.5436457395553589, -2.6698639392852783, 0.36877989768981934, -1.423702359199524, -0.8094850778579712, -0.6001595854759216, -0.5109844207763672, -0.625286340713501, 0.5910007357597351, -0.6325874328613281, 0.2260732501745224, -1.711690068244934, 0.7115616202354431, -0.869610071182251, -0.3849310278892517, 1.6123602390289307, -1.0076082944869995, 1.176888108253479, -0.7531079053878784, -0.1145649179816246, -0.065247543156147, 0.5948442220687866, -0.3356582224369049, 0.17770250141620636, -0.5393252968788147, -0.7632374167442322, 0.2950003445148468, -1.186705231666565, -2.3643524646759033, 0.7564008235931396, 0.06313513219356537, 0.29767271876335144, -0.8332754969596863, 1.6609108448028564, 1.3252402544021606, -0.7850885987281799, 1.9402055740356445, 0.9943616390228271, 1.0034887790679932, 0.9353810548782349, 0.5353962182998657, -0.1388646513223648, 0.3095090389251709, -1.220653772354126, 0.6949639916419983, 0.10417111217975616, -0.11842963099479675, 0.09951461851596832, 0.6691300272941589, -0.3649466931819916, -1.7502554655075073, 0.12020175904035568, 0.007616677787154913, -1.8054839372634888, 0.6622791886329651, 0.4876692295074463, 0.07942572981119156, -0.08544319123029709, -1.6455577611923218, 1.2851152420043945, -0.27018749713897705, -0.5880494713783264, -0.23789814114570618, -0.7613175511360168, 0.8662620782852173, -0.8190483450889587, -0.11211923509836197, -0.5843197703361511, -0.43488582968711853, -0.6430534720420837, 0.21785977482795715, -0.4881366193294525, -0.617274820804596, 0.5549508929252625, -0.7851637601852417, -1.904347538948059, -0.9919559359550476, 0.3786585032939911, 0.3115153908729553, 0.5340389609336853, 0.9814955592155457, -0.4538213908672333, -0.45672357082366943, -1.6983637809753418, -0.14937378466129303, 0.8097910284996033, 0.5936262607574463, 0.8459823131561279, -1.0711718797683716, -0.8104761838912964, 0.2227039486169815, 0.5903923511505127, -0.3361258804798126, 1.886838436126709, 0.2619883418083191, 0.6298548579216003, -1.1350194215774536, 0.1054229587316513, -0.08346482366323471, -1.8502185344696045, -2.0640182495117188, 1.1578670740127563, 0.4875335395336151, 0.047917142510414124, 0.6003192067146301, 1.163288950920105, 0.47489774227142334, -1.418738842010498, -0.3763933479785919, 0.2782863974571228, 0.20999860763549805, 0.009610430337488651, 0.8583588600158691, -0.396067351102829, 0.009701462462544441, 1.1436434984207153, -0.4339507818222046, -0.1257198303937912, 0.7832661271095276, -1.2700034379959106, -1.2026859521865845, 2.2227935791015625, 0.7200839519500732, -0.6409904360771179, -1.7237210273742676, -0.57103431224823, 0.8269751071929932, 0.32434242963790894, -0.6550684571266174, 0.8031864762306213, -0.19335748255252838, 0.15098097920417786, -3.1699411869049072, 0.6568830013275146, -0.21736979484558105, -1.1617521047592163, 1.297649621963501, 0.7584648728370667, 0.5560600757598877, -1.1945528984069824, -0.3733169138431549, -0.4568704664707184, -1.2994084358215332, -0.2470986396074295, -1.3269824981689453, -0.41998058557510376, 2.066056728363037, 0.7308018207550049, -2.3873438835144043, -1.6859173774719238, -0.8527390360832214, -1.6014589071273804, 1.1680058240890503, -0.7296721339225769, 0.033263299614191055, 0.3404921889305115, 1.0540510416030884, 0.8213186860084534, -0.09927859902381897, 0.7223270535469055, 1.0614187717437744, 0.02351580373942852, -1.3382080793380737, 0.6459285020828247, -0.2506703734397888, 1.434026837348938, 0.20614992082118988, 1.3818657398223877, -1.366731882095337, 1.8982436656951904, -0.39915910363197327, 0.27450042963027954, 0.4430287182331085, -0.750014066696167, -0.16537737846374512, -1.4498004913330078, 0.5927731990814209, 0.4719095230102539, -0.2090097963809967, -0.6711238622665405, -1.0435941219329834, 1.3534104824066162, 0.2340220957994461, 1.0596392154693604, -0.2317296713590622, 0.6854186654090881, 1.0379905700683594, -0.12855088710784912, -0.4572474956512451, -0.5253722667694092, -0.22069045901298523, -0.20618940889835358, -0.7511611580848694, 0.5358461737632751, -0.7351654171943665, 1.2977179288864136, 0.9244973063468933, 0.02140362188220024, 1.2024304866790771, 0.35629138350486755, -0.1721855252981186, 0.02950735203921795, 0.7104905247688293, -0.8418795466423035, -0.5825818181037903, 0.4456782042980194, 1.4753572940826416, 0.2731541693210602, -0.8474131226539612, 1.5231164693832397, -0.6149231791496277, 0.050837986171245575, 1.685808539390564, -1.1901676654815674, 0.7705786824226379, -0.6804178357124329, -0.7608146667480469, 0.17414455115795135, 0.199508935213089, -0.22032642364501953, -1.0263919830322266, -1.102543592453003, 0.3257097899913788, 0.944563090801239, -0.532850444316864, -0.9084128141403198, -0.7782092690467834, -0.7491604685783386, 1.3352760076522827, 1.067240595817566, -1.6888467073440552, -0.7841324210166931, 0.5508762001991272, -0.2452690750360489, 0.4295102655887604, -0.5055557489395142, -0.2086670994758606, -0.11225064098834991, 0.3546845316886902, -0.12978282570838928, 0.40202581882476807, -1.376932144165039, -0.957554817199707, 0.25996339321136475, -0.906624972820282, -0.8639616370201111, -0.2246686965227127, 1.3531073331832886, -1.403739094734192, 1.5682939291000366, -0.5695368051528931, -0.5640569925308228, -1.3407622575759888, -0.05211472511291504, -0.3295222520828247, -0.21705999970436096, 0.2147667557001114, -0.6589066982269287, 0.2980213165283203, 0.22371114790439606, -0.8838905096054077, -1.6967147588729858, -0.27919602394104004, -0.2181323617696762, -0.9643059968948364, -0.9635829925537109, -0.03220610320568085, 0.8164680600166321, -0.6790720820426941, -1.6903878450393677, -0.3322719633579254, -0.8141079545021057, -1.0802372694015503, 1.1705446243286133, 1.1876381635665894, 0.9506700038909912, 2.276273250579834, -0.0817047581076622, -0.3693219721317291, 0.21390998363494873, -1.649153232574463, -1.8113707304000854, 0.6728178262710571, -1.3448045253753662, -0.23945580422878265, 0.2783791720867157, -0.19677934050559998, 0.9565163254737854, 0.7220972180366516, -0.6941556334495544, 0.6059070825576782, 1.134027361869812, 0.8070274591445923, 0.14125439524650574, 0.4449797570705414, 0.2960250973701477, 0.9904909729957581, 0.42767658829689026, -0.375656396150589, -0.1198854073882103, 0.6155079007148743, 0.4784902036190033, 0.1893693059682846, 1.0373448133468628, -0.0725630670785904, 0.33640098571777344, -1.5956358909606934, 0.07402635365724564, 2.1039161682128906, -0.7048168182373047, 0.2528216242790222, 0.5154833793640137, -1.0488300323486328, 0.8974987864494324, -0.008265828713774681, 1.0820103883743286, -2.508151054382324, 1.1277964115142822, 0.17455758154392242, 0.06458469480276108, 2.0248072147369385, -1.316466212272644, 1.1700739860534668, -0.38095444440841675, 2.9343173503875732, 0.4163517355918884, -0.22338193655014038, -1.4102526903152466, -1.4668480157852173, -0.6976863741874695, 0.10042884200811386, 1.468869924545288, -0.1945546567440033, -1.2537548542022705, -0.9564420580863953, 2.40067458152771, 1.0230273008346558, -0.15609429776668549, 1.223861575126648, 0.6700602769851685, 0.2751395106315613, 1.0427502393722534, -0.7567501068115234, 0.0008901637047529221, -0.24487100541591644, 2.3102591037750244, 0.11738769710063934, 0.4449651539325714, 0.25575217604637146, 0.5435276627540588, 0.9527140259742737, -0.007696635089814663, 0.5071783661842346, -1.1927145719528198, -0.28980499505996704, 1.2437433004379272, 1.4672354459762573, -0.9272409081459045, -0.6179789900779724, -0.0797484964132309, 1.3571839332580566, -0.7420791983604431, 1.1644728183746338, -0.26974278688430786, 0.3622053861618042, 0.30524685978889465, 0.7161481976509094, 1.0046758651733398, 1.8738746643066406, 0.8604071140289307, -1.5330396890640259, -0.09934485703706741, 0.16024567186832428, 0.3440055847167969, 0.30702096223831177, -0.0793878585100174, -0.07156872749328613, -0.5430729985237122, 0.3418837785720825, -1.3498557806015015, -0.5314999222755432, -1.1860982179641724, 2.211329936981201, -0.5094251036643982, 0.5216048955917358, 1.140365719795227, -0.3851519525051117, -0.6859299540519714, 0.6230104565620422, 2.4970016479492188, 0.07844450324773788, -2.1407103538513184, -0.7069282531738281, 1.735047459602356, 1.5486423969268799, -1.0959017276763916, 0.02195335179567337, 0.858371913433075, -1.5928823947906494, 1.2852340936660767, -0.141592338681221, -1.3209402561187744, -0.8273783922195435, 0.44135400652885437, -0.34020519256591797, -1.8383057117462158, 0.1147073358297348, 1.2975984811782837, 1.1247514486312866, -0.9790778756141663, 0.960175096988678, 1.6194039583206177, -0.8348348736763, -1.407153606414795, -2.7350807189941406, -1.727606177330017, 1.3028945922851562, 0.4254809319972992, 0.06899433583021164, 0.37317416071891785, -0.8358579277992249, -1.6806598901748657, 0.7018633484840393, 0.9427449703216553, 0.5900296568870544, 1.288482904434204, -0.5023730397224426, -0.08381976187229156, -0.47869306802749634, -0.7376790046691895, 0.7467674016952515, -2.197382688522339, -0.8891628384590149, -1.0388115644454956, -1.3251769542694092, 0.1656446009874344, 0.5885867476463318, -0.15658411383628845, -1.9698177576065063, -1.2154865264892578, 0.5352485775947571, -0.4263322651386261, -0.8603392839431763, -1.574870228767395, -0.7431918382644653, 0.7457712292671204, -0.42029622197151184, 0.23471465706825256, -2.2576327323913574, -0.3661631643772125, -1.455774188041687, -1.4135996103286743, 1.114262580871582, 0.09692263603210449, 1.2880885601043701, 1.0135557651519775, 2.8985395431518555, -1.468684434890747, 0.5324065685272217, -0.040062759071588516, 0.9557909965515137, 0.032666031271219254, 0.06385023891925812, 0.4471406936645508, -0.3684926927089691, 0.8335211277008057, -0.1477198302745819, 1.301435947418213, 0.8702020645141602, 0.43777379393577576, 0.4922126829624176, 0.16635042428970337, -1.1875020265579224, 2.1218302249908447, -1.4211453199386597, 0.6667178869247437, 0.5098599791526794, -0.0003575475129764527, 0.28684133291244507, 0.5993054509162903, 2.2602810859680176, 1.654679775238037, 0.8070515394210815, 1.2328217029571533, 0.4655342400074005, -1.7661010026931763, 0.6274648308753967, 2.0575857162475586, 0.8456820249557495, -0.3436845541000366, 0.43434733152389526, 1.2137495279312134, -2.0713729858398438, 1.0184839963912964, 0.6345755457878113, 1.3394489288330078, 0.673709511756897, 0.8094244003295898, 0.9970890283584595, 1.24326491355896, -0.02257039025425911, -0.22337687015533447, -0.21599379181861877, 0.7844063639640808, -1.068823218345642, -2.3208980560302734, -0.6365454792976379, -2.8297653198242188, 0.8761689066886902, 0.08051767200231552, -0.9939300417900085, -0.45807257294654846, 0.5927792191505432, 0.9961167573928833, -1.354668378829956, -0.17871429026126862, -0.2571994662284851, -0.7109131217002869, 1.3294317722320557, 0.7204824686050415, 0.5035048127174377, -1.3875294923782349, 0.6195911169052124, 0.3891780972480774, -0.853884756565094, 0.5077825784683228, 1.504633903503418, -0.022657359018921852, 1.417837381362915, 1.0052499771118164, 0.9464318752288818, 0.8287037014961243, 1.300402045249939, -0.03521289676427841, -1.3758810758590698, -0.3828336298465729, 0.2792213559150696, 0.5115564465522766, 0.1865285187959671, 0.26719826459884644, 1.526348352432251, -1.601892352104187, -1.3703891038894653, -0.9849548935890198, 0.9196613430976868, -0.2838645577430725, -1.3370281457901, 0.5157667994499207, -0.6372228860855103, -1.5125099420547485, 0.43330734968185425, 0.5027331709861755, 0.48353999853134155, -0.6250665783882141, -0.5103533864021301, 1.877631664276123, -0.305388867855072, -1.1764066219329834, 1.3393115997314453, -1.4926029443740845, -0.49897611141204834, 0.8555165529251099, -0.5884608030319214, -0.7907673120498657, 1.0888550281524658, -0.4222724735736847, 1.478177547454834, -0.638531506061554, 0.11522794514894485, -1.278562068939209, -0.07434172183275223, 1.6457808017730713, -0.2520420551300049, -0.16175664961338043, -2.238703727722168, 0.4639377295970917, 0.7655768990516663, -0.1802908480167389, 0.45680224895477295, -0.12750917673110962, 1.1208163499832153, 1.8017021417617798, -0.5534266829490662, 0.03685276582837105, 1.3021682500839233, 0.1818857491016388, 0.07665056735277176, -1.0983713865280151, -1.1560990810394287, -0.11418940871953964, 0.5267359018325806, -0.6715699434280396, 0.21328884363174438, -0.7410299777984619, 0.2160666584968567, -0.15670479834079742, -0.17347340285778046, -1.1145949363708496, 0.30226585268974304, 0.47835907340049744, -0.7537699937820435, -2.544057607650757, 2.336310386657715, 0.3958825469017029, -1.1560711860656738, 1.0361812114715576, 0.14429064095020294, -1.2180720567703247, 0.20573623478412628, -0.3795812427997589, -0.20830191671848297, -0.49101126194000244, -0.32948559522628784, -1.3804919719696045, 0.41687238216400146, -1.0613237619400024, -1.553357481956482, -1.2473928928375244, 0.45317766070365906, -0.20092065632343292, 0.17024332284927368, -1.0122777223587036, 0.11504346877336502, -0.07146527618169785, 0.9311330914497375, 0.5963345170021057, -1.5559593439102173, -0.9719608426094055, -0.8371780514717102, -0.518077552318573, -1.51972496509552, -1.3500913381576538, 0.07637033611536026, -0.4852587580680847, 1.5751549005508423, 0.11676563322544098, 0.12138965725898743, -2.057642936706543, 0.5821076035499573, 0.6603225469589233, 2.0737180709838867, 0.09995195269584656, -0.41720354557037354, -0.4856584668159485, 2.0902650356292725, -0.30264100432395935, 0.05149880796670914, -1.702970266342163, 1.182881474494934, -1.2274843454360962, -0.13628530502319336, 0.28052380681037903, 0.6553584933280945, -0.06661362946033478, 0.11077366769313812, -0.23055584728717804, 1.071448564529419, -0.18255384266376495, 0.07956910878419876, 0.9034678936004639, 1.1316890716552734, 0.36229419708251953, -0.14430604875087738, -0.5816907286643982, 0.25229504704475403, -1.1853357553482056, 0.27762553095817566, 1.3368136882781982, -0.541079580783844, -0.1712752878665924, 0.03017568588256836, -0.8726778030395508, -0.9468714594841003, -0.6193222403526306, 0.947350800037384, 0.5643394589424133, -0.3896450996398926, -1.821768045425415, 0.8839137554168701, 1.137841820716858, -0.3976629078388214, -0.9992395043373108, 0.30199751257896423, 0.7457131147384644, 2.0106685161590576, -1.328290581703186, 0.7153377532958984, -1.8342043161392212, 0.26240071654319763, 0.2933062016963959, 2.1272683143615723, 0.840141236782074, -0.931937038898468, -0.8939610123634338, -0.21995234489440918, 2.6857404708862305, -0.2646377384662628, -1.2972036600112915, -1.2436202764511108, -0.9188977479934692, 0.02795327641069889, -0.8334100246429443, -1.2101972103118896, -1.2691445350646973, -0.2895181477069855, 0.06886085122823715, -1.006952166557312, 0.6032251715660095, -0.9811163544654846, -0.23205901682376862, -0.9986409544944763, -0.19437116384506226, 1.2237164974212646, 0.9904327988624573, 0.9885666370391846, 0.8399123549461365, -0.050025880336761475, 1.651383399963379, 0.1635856032371521, 0.4783081114292145, -1.378137230873108, -0.2528259754180908, -0.8268749117851257, 1.3764376640319824, -1.5847362279891968, 0.9063357710838318, -1.0514591932296753, 2.2456235885620117, -0.008933540433645248, 1.203688383102417, 0.4579136073589325, -1.7179499864578247, -0.48525527119636536, -1.4179548025131226, -0.22028757631778717, 0.012339930050075054, -1.6961082220077515, -0.8552852869033813, -0.4247943162918091, 0.36746811866760254, 1.091894268989563, 0.5554897785186768, -0.10036813467741013, 0.1165221631526947, 0.4704171419143677, -0.8084129095077515, 1.8276562690734863, -1.1010879278182983, -0.009257890284061432, 0.42924776673316956, -0.7373512387275696, 1.512613296508789, 0.5936731696128845, -2.092247724533081, 0.20726269483566284, -0.9991148114204407, 0.747035026550293, -0.2033764123916626, 0.19498009979724884, 1.1051011085510254, -0.678150475025177, -0.8886451721191406, 0.7862738370895386, -1.1743130683898926, -0.9463304281234741, -0.008014548569917679, 1.591583013534546, -1.1893409490585327, -2.6072819232940674, 0.6296669840812683, -2.039966344833374, -0.2097536325454712, 0.28500014543533325, 0.8400671482086182, 1.3933758735656738, 0.9796978831291199, -0.8634160161018372, -0.6646313071250916, 0.6194698214530945, 0.7761358618736267, -0.4795273542404175, 0.9219140410423279, -0.6445506811141968, 0.0965283215045929, -0.3245575726032257, -0.4239865839481354, 0.5381001234054565, 1.8973289728164673, 0.9119618535041809, -2.3671441078186035, 1.1400271654129028, 0.14850397408008575, -0.8956253528594971, -0.1645987182855606, 2.489638328552246, 1.8056126832962036, 0.9240297079086304, 0.8594178557395935, 1.1862518787384033, 1.1460305452346802, 0.5763375163078308, 0.0628318190574646, -0.6401039361953735, -0.12442804127931595, 0.3839300274848938, 0.9995989203453064, 0.8300654292106628, 0.1559494137763977, -0.7237347960472107, 1.419878363609314, 0.0805356353521347, -0.016276920214295387, 0.18428584933280945, -0.34418344497680664, -0.48846912384033203, 0.08385193347930908, -0.9995925426483154, 0.2520887851715088, -0.4495188891887665, -0.714368462562561, -1.6096525192260742, 0.24292615056037903, -1.0906562805175781, 0.2237711101770401, 3.392482042312622, -0.04017909616231918, -0.896144688129425, 1.1369616985321045, 0.3308452367782593, 0.5866461396217346, 1.6907260417938232, -0.7688166499137878, 0.5391745567321777, 1.1359552145004272, -1.6213361024856567, -0.12022756785154343, 1.3369145393371582, -0.3407547175884247, -0.19013535976409912, -0.4173032343387604, 0.2452271729707718, 1.3023459911346436, 0.828515350818634, 1.3941996097564697, -0.6272862553596497, 0.09485014528036118, 0.9878480434417725, -0.8412151336669922, -0.7008926272392273, -0.4009977877140045, -0.5989258289337158, 1.4919450283050537, -0.5096731781959534, -0.3153894245624542, -1.277775764465332, 0.6563843488693237, -2.3626201152801514, 0.8572738766670227, 0.24629288911819458, -0.4839555323123932, -2.1566317081451416, -1.0871367454528809, 0.9277425408363342, 0.7141158580780029, 0.7701665163040161, 0.8339408040046692, -0.014636955223977566, -0.4545230567455292, -0.13063783943653107, 0.7337015867233276, -0.9418885111808777, 0.701235294342041, 0.9934858083724976, 2.107915163040161, 0.9592016339302063, 0.3144737482070923, -0.6351008415222168, -0.6032746434211731, 1.1093885898590088, 0.3239007890224457, 0.6066889762878418, 0.8474708795547485, 0.054801467806100845, -0.23252888023853302, 1.0979855060577393, 0.8986053466796875, -0.7151235342025757, -1.039422631263733, -2.7200138568878174, -0.3741566836833954, 1.1503382921218872, -1.3137788772583008, 0.7211779356002808, 1.7246017456054688, 0.4137319028377533, -0.22844111919403076, 0.04021511599421501, 0.2633401155471802, 1.7291699647903442, -0.48185333609580994, -1.7673393487930298, 1.0209654569625854, -0.5448335409164429, 0.41928744316101074, 0.5584900975227356, -0.11886971443891525, -1.1601009368896484, -2.467358112335205, -0.08912578225135803, 0.31035923957824707, -1.5812671184539795, -0.9270511865615845, -0.16875889897346497, 0.6234499216079712, -0.20839522778987885, 0.4281715750694275, 0.6845949292182922, 0.5238238573074341, -0.3863276243209839, 0.5229882001876831, -0.38244977593421936, -1.7158379554748535, 0.4058424234390259, -0.8128694891929626, 0.46671274304389954, 0.7427266240119934, -1.7001020908355713, 0.8074484467506409, 0.2482541799545288, -0.9825460314750671, 0.004349295515567064, 0.9021012187004089, -0.01541372761130333, 0.6256003975868225, 1.4539504051208496, -0.1692575365304947, 0.4387105405330658, -0.8683600425720215, 0.18831421434879303, -0.5220482349395752, 0.7114462852478027, 0.11905500292778015, -0.9269334077835083, 0.7388874888420105, -1.0073291063308716, 0.8117632865905762, 1.1354559659957886, 0.821942925453186, -1.787479281425476, 1.0722237825393677, -1.220502495765686, -0.47441816329956055, -1.2576258182525635, -1.1247848272323608, 0.18835857510566711, -0.4025470018386841, 1.086846113204956, 0.009436503984034061, 0.9505994915962219, 0.9936463832855225, 0.7920960187911987, -0.29884108901023865, -0.18889817595481873, 0.47286227345466614, -0.5059224963188171, -0.1765560507774353, -0.6900700330734253, 0.6860466003417969, 0.2594875395298004, 0.6529930830001831, 0.30905523896217346, 0.3743915557861328, -0.9704765677452087, 2.1180665493011475, 0.9185088276863098, 1.0744025707244873, -1.5607162714004517, -1.211207628250122, -0.4914986789226532, -1.8246973752975464], [-1.7384819984436035, 1.887435793876648, -0.622545599937439, 0.9320598244667053, 0.6747128367424011, -0.6683615446090698, 1.1696454286575317, 0.81170654296875, 1.1234246492385864, 0.6740751266479492, -0.9922933578491211, -0.6334582567214966, 0.9110795855522156, -0.17451788485050201, 0.4643233120441437, 0.06855658441781998, 1.3101423978805542, 1.711807370185852, 0.5698632597923279, -1.009600281715393, 0.6725963354110718, 0.3020249605178833, 0.1820695549249649, 1.4802383184432983, 1.2383460998535156, -0.2503674328327179, 0.8009219765663147, -0.8703137040138245, -1.0747560262680054, -0.6014275550842285, 0.19744180142879486, -2.031805992126465, 0.9756060838699341, 0.8067182898521423, 1.220022439956665, -0.6744667291641235, 1.6437416076660156, -0.3731975555419922, -1.028806447982788, 0.2907675802707672, -0.12114715576171875, 0.8734362125396729, 0.2768360674381256, -1.0251219272613525, 1.0062099695205688, 0.533511757850647, -0.01905585639178753, 0.28859153389930725, 0.05168446525931358, 0.6860297322273254, -1.49684739112854, -0.5503446459770203, -1.2860783338546753, 0.6966907978057861, -0.2982092499732971, -0.3466586172580719, -1.3857117891311646, 1.5430234670639038, 1.5498459339141846, 0.3699762225151062, -0.01652207039296627, -1.0182322263717651, 0.99619060754776, 0.10055223852396011, -0.046831220388412476, 0.5329217314720154, -0.3231654763221741, -1.557060718536377, 0.48843497037887573, 1.069925308227539, 1.259229302406311, 1.2446682453155518, 0.7493820190429688, -0.04864932969212532, 1.269335389137268, -0.9494214653968811, -0.1969257891178131, -0.3666476011276245, 0.5983714461326599, 1.5137686729431152, -1.3909893035888672, -0.1460253745317459, 0.17113734781742096, -1.1301753520965576, -1.241720199584961, 0.12338301539421082, -1.0249521732330322, 0.06639768928289413, 2.609170913696289, -0.5496354103088379, 0.8600612878799438, 0.3828856348991394, 0.06877327710390091, -0.8239372968673706, 0.17673857510089874, 1.0922067165374756, -1.4621658325195312, -0.7316643595695496, 1.0047107934951782, -0.7655336856842041, -2.240135908126831, 0.043480124324560165, 1.1126887798309326, -0.7982730269432068, 0.24239487946033478, 0.8955176472663879, 0.21352122724056244, -1.2975270748138428, -1.9297897815704346, -0.004556889645755291, 1.546833872795105, 1.1931474208831787, 0.40171360969543457, 0.31320908665657043, 0.7873401045799255, 1.0062919855117798, -0.09423469752073288, 1.235521674156189, -0.054413266479969025, 1.4766976833343506, -2.4559872150421143, 1.628240942955017, 1.7778289318084717, 1.1009385585784912, -0.8810030817985535, -2.998664379119873, 0.046658582985401154, 0.07502510398626328, -0.21206209063529968, 1.0019136667251587, 0.41127821803092957, -1.348400592803955, -0.5358035564422607, 0.5783098936080933, -1.4447729587554932, -0.8802754878997803, -0.6401434540748596, -1.150651454925537, -0.3873273730278015, -1.5116400718688965, 0.4331689476966858, -1.5988197326660156, 0.8772252202033997, 2.416999340057373, 0.10206560045480728, -0.22959314286708832, -1.3749905824661255, 0.8902854323387146, -1.5137122869491577, -1.5182981491088867, -1.827957272529602, 0.84944748878479, 0.3770940899848938, -0.8220384120941162, -0.590422511100769, 0.0387921966612339, 0.16466252505779266, -0.1035490483045578, 0.8445013761520386, -0.5338481664657593, 0.8233698010444641, -0.7162451148033142, 0.5096728801727295, 0.397167444229126, 1.1548570394515991, 1.4691096544265747, -0.47079429030418396, 0.1137392595410347, 1.39098060131073, 0.3929532766342163, 0.8749643564224243, -0.09218849986791611, 0.3481549918651581, 0.5839821100234985, 1.0075430870056152, 1.3654282093048096, 1.4353147745132446, 0.1241183876991272, -0.9684534072875977, -1.3292114734649658, -2.039233922958374, -0.6144980192184448, 1.7781956195831299, -1.2220898866653442, 0.5336214303970337, 0.5992177724838257, -0.01184510625898838, -0.07975643873214722, 0.6131750345230103, 1.370706558227539, -0.15787950158119202, 0.5148717761039734, 1.5501943826675415, 0.461760014295578, -0.25334790349006653, -0.3429062068462372, -0.8051708936691284, 0.5230527520179749, -1.2832225561141968, 0.8990864157676697, 0.07168135046958923, 0.33852770924568176, -0.3058672249317169, 0.8324660658836365, 0.6524391770362854, 1.6375038623809814, -0.17482633888721466, 1.877966046333313, 1.8575385808944702, 0.26843398809432983, 1.3833866119384766, 0.5816195607185364, 1.1372642517089844, 1.4312442541122437, -0.6964478492736816, -0.9720995426177979, -0.1401861160993576, -1.2165628671646118, -1.5264497995376587, -1.1363123655319214, -0.8776893615722656, 0.03880852088332176, -0.2977687418460846, -0.6623210310935974, 0.4063572883605957, 0.3210359811782837, -0.23302023112773895, -0.8550969362258911, -0.11481723934412003, 0.4787716865539551, -0.46131762862205505, 1.739785075187683, 1.9775583744049072, -2.175196409225464, -0.1479576975107193, 1.1188066005706787, -1.9950733184814453, 1.260779619216919, -0.32983314990997314, 1.2954562902450562, -1.3262017965316772, -1.6248962879180908, 0.4760797321796417, -1.4105550050735474, -0.5441952347755432, 1.396012783050537, -0.2437342256307602, -0.36372166872024536, -0.1281033605337143, 1.1390992403030396, 1.5961533784866333, -1.6477380990982056, 1.1874028444290161, 0.595724880695343, 0.731768012046814, 0.6150174140930176, 0.8338047862052917, 0.24495014548301697, -0.02380317635834217, -0.8869641423225403, 0.29588499665260315, -1.7178946733474731, -0.6447324156761169, -2.252819538116455, 0.9448792934417725, 1.1659682989120483, 0.8855583071708679, 0.7693087458610535, 0.5134423971176147, -0.6245647072792053, 0.8862813115119934, 0.996135413646698, -0.01926450990140438, -0.26149794459342957, 1.6370456218719482, -0.18256503343582153, 0.6641350984573364, 0.3571545481681824, 1.0948309898376465, 1.106386661529541, 0.6266909241676331, -0.1583641767501831, 0.30664902925491333, -1.0494962930679321, 1.2617591619491577, -0.03063919022679329, -0.795302152633667, 1.2821927070617676, 0.4416906535625458, -0.16374820470809937, -1.2438751459121704, -0.8901117444038391, -0.6506602168083191, -0.2709948420524597, -1.221240758895874, 0.6548434495925903, -1.7682621479034424, 0.3373631536960602, -0.36566784977912903, -0.05995076522231102, 1.8339115381240845, -1.5953291654586792, 0.11620300263166428, 0.4014607071876526, 0.6183794140815735, 1.609147310256958, -0.6366592645645142, 0.13664287328720093, 0.1336495578289032, 0.4915202856063843, 0.2871493399143219, -1.3586007356643677, -1.4662189483642578, 0.2922528386116028, -0.3248392343521118, -0.2826935648918152, 0.82197105884552, 0.8554961085319519, -0.17923802137374878, 0.02509159967303276, 0.26750868558883667, -0.8487960696220398, -0.2655741572380066, -0.2494562417268753, 0.38356128334999084, -0.5508983731269836, 0.8819957375526428, -2.121053457260132, -0.7317572832107544, 0.48024699091911316, 2.599652051925659, -1.6932568550109863, 0.5866772532463074, -0.2306007295846939, -1.0753822326660156, -0.5560439229011536, -0.05307984724640846, 0.0531192310154438, -0.49935609102249146, 1.474896788597107, -0.1655462235212326, 0.3967570662498474, -0.33307206630706787, -0.5088837146759033, 0.04446438327431679, -1.038826584815979, 0.33694198727607727, -0.6196498274803162, 0.6767671704292297, 0.05837167054414749, 0.41049474477767944, -1.1027456521987915, -1.247811198234558, 0.8244602084159851, -0.3700183033943176, -2.2897071838378906, 0.909788966178894, 0.556709349155426, 0.02230634167790413, -0.34599459171295166, -0.038547199219465256, -0.6895943880081177, 0.18835745751857758, -0.871617317199707, -0.3930610120296478, 1.8684858083724976, 1.2983973026275635, 0.7986117005348206, 0.7520315051078796, 1.9228615760803223, 0.787144660949707, 0.28211700916290283, 1.4309337139129639, 0.23924119770526886, 0.12532702088356018, 0.7065075039863586, 0.5311046838760376, -0.029178645461797714, -2.2321255207061768, 0.6792123913764954, -0.06290825456380844, -0.7060852646827698, -0.5152122378349304, 0.8388882875442505, -1.762969732284546, -0.7718175649642944, -0.9222208857536316, 0.6399633884429932, -0.2725895345211029, 0.028790190815925598, 1.71100914478302, 1.0721914768218994, 1.0187472105026245, 1.2476638555526733, 0.2236352562904358, 0.28290873765945435, 2.8827590942382812, -0.2287834882736206, -1.039165735244751, -0.967340350151062, -0.1812606006860733, 1.0602891445159912, 0.8846536874771118, -2.511798858642578, -0.7544565200805664, -0.2963476777076721, 0.23268555104732513, 0.2706872820854187, 0.2666270434856415, 1.803842544555664, 0.9212205410003662, 0.14387473464012146, -0.11461655050516129, -0.7966893315315247, 0.36647409200668335, 0.5259041786193848, -1.9936714172363281, -1.2119125127792358, -0.7271857857704163, -0.2860768139362335, -0.4675499200820923, -0.6897292137145996, 0.242966890335083, 0.3115306496620178, -0.8333112001419067, -0.1452322006225586, 0.2468823790550232, 1.6625654697418213, 1.0154746770858765, -0.4351954162120819, -0.6874912977218628, 1.09029221534729, -0.0454675666987896, 2.329576253890991, 1.2131679058074951, 2.0273048877716064, 0.9285238981246948, -1.343816876411438, -0.6394128799438477, -0.44710901379585266, 0.9566806554794312, 0.9935951828956604, -1.2026718854904175, 0.2636966407299042, 1.2746137380599976, 0.9609590768814087, -0.19655798375606537, -0.6971738338470459, 0.6931502819061279, 0.20080365240573883, 1.3281468152999878, -0.4632667303085327, 1.0017995834350586, -0.45524194836616516, 0.1183418333530426, -0.17554253339767456, -1.6442131996154785, -1.3903509378433228, -0.5225174427032471, 0.24007248878479004, 0.4123417139053345, -0.5940362811088562, -0.3332028388977051, 1.9882954359054565, 0.45470374822616577, 0.33908766508102417, -0.3652607202529907, -1.8391494750976562, 0.1426415741443634, -1.1708730459213257, 1.318523645401001, -0.6450578570365906, -0.5874078869819641, -0.08183080703020096, 0.3576553761959076, -0.0981970950961113, 0.5929957628250122, 1.1284962892532349, 1.2394826412200928, 0.3200021982192993, -1.260025978088379, -0.6318796873092651, -1.9812692403793335, -0.655407726764679, 0.451189786195755, 0.003235467476770282, -1.0658111572265625, 1.2178466320037842, -0.3617905378341675, 0.18036657571792603, -0.8951670527458191, 0.43911781907081604, 0.7404034733772278, 0.23573285341262817, -0.06389527767896652, 0.8497010469436646, -0.8285611271858215, 0.03715340793132782, 0.6413280367851257, 0.6090689301490784, 0.5475909113883972, 1.5523109436035156, -0.9669823050498962, -0.7258901000022888, 0.23280160129070282, 1.0000232458114624, 0.6989096403121948, -1.238264560699463, 0.38633617758750916, -0.060140375047922134, 0.22971735894680023, 0.6287325620651245, -1.427380084991455, 0.35051411390304565, 0.6634105443954468, -2.481497287750244, 1.1065385341644287, -0.14174188673496246, -0.4831787049770355, 0.6481671929359436, -0.9976251721382141, 0.10295426845550537, -0.6025926470756531, -0.26727721095085144, -0.06024303659796715, 1.8261244297027588, 0.09322716295719147, 2.6497199535369873, 0.6462125182151794, 0.8165847659111023, -0.440385639667511, -0.9533456563949585, 0.4379351735115051, 1.3754223585128784, -1.281574010848999, -0.8685376644134521, 1.4874835014343262, -1.6663225889205933, -0.8395628333091736, 0.8580166697502136, -0.3800624907016754, -2.036379337310791, 0.44126564264297485, -0.9639207124710083, -0.30805954337120056, -1.4529246091842651, 0.11249138414859772, 0.25865763425827026, 0.6754409670829773, -1.7953345775604248, 1.305985450744629, 0.09212259948253632, -1.2725133895874023, -1.3549593687057495, 0.006921037565916777, -0.3946355879306793, 1.3258509635925293, 1.0736140012741089, -0.4472997188568115, -1.086736798286438, -0.6255626082420349, -1.3445401191711426, -0.636673092842102, -1.2938950061798096, -2.4752566814422607, 2.2846875190734863, 0.12786200642585754, -0.8016610145568848, 0.6307616829872131, 0.6745150089263916, 0.07129599153995514, 0.22700372338294983, -0.8658422231674194, -1.2063231468200684, 0.47846752405166626, 0.7854648232460022, -2.0482122898101807, 0.24751465022563934, -0.09147461503744125, -0.6918373107910156, 0.5247107744216919, 0.058020997792482376, 0.9681094288825989, 1.1760241985321045, 0.013881128281354904, 0.6109296083450317, 0.06534277647733688, -0.05607895180583, 0.3880925178527832, 0.2795054316520691, -0.24724224209785461, 1.2110096216201782, 0.9802184700965881, 0.22951051592826843, 1.3543719053268433, 0.9427220821380615, 0.1110183447599411, 0.0922425389289856, 0.8427775502204895, -2.7924838066101074, -0.8976714015007019, -1.7251101732254028, 1.4453777074813843, -0.31764310598373413, 1.8416874408721924, -2.087552309036255, 1.7860511541366577, 0.31461623311042786, -0.22472836077213287, -0.5132953524589539, -1.0936082601547241, -1.013414740562439, -1.153399109840393, 0.4953649640083313, -0.28156691789627075, -0.21042831242084503, 0.3540323078632355, -1.2246644496917725, 0.7212283611297607, -1.5797885656356812, 1.2624174356460571, 0.19610324501991272, 0.39799821376800537, -0.412370890378952, -0.1133347898721695, -1.0655204057693481, -1.7943098545074463, -1.1009589433670044, -0.33797016739845276, -0.9098185300827026, 0.4976752698421478, 0.6762403845787048, 0.2048267275094986, -0.21366740763187408, 0.47787511348724365, -0.5871549248695374, -0.5561118125915527, 0.5768506526947021, -0.6698682308197021, -0.9341049194335938, 0.10478297621011734, -0.8020622134208679, 0.18304148316383362, 0.27034687995910645, -0.06781607121229172, -3.420821189880371, 0.6174601912498474, -0.9418602585792542, -0.3983049690723419, 0.40899959206581116, 1.9353141784667969, 0.0711510106921196, -0.22327837347984314, -0.6688553690910339, 0.3274000287055969, 0.6177514791488647, 0.1502680480480194, 1.076499104499817, -0.5911561250686646, 0.47957944869995117, -0.6228082180023193, 0.5281685590744019, -0.4549328088760376, -0.13057273626327515, 2.1268198490142822, -0.31840023398399353, -0.15352895855903625, -1.0718142986297607, -0.44978073239326477, 0.7872628569602966, -0.5210393071174622, 2.3760757446289062, 0.3778020441532135, -0.04012659564614296, -0.07020359486341476, 0.8357774615287781, 0.9814916253089905, 0.017242038622498512, -0.34592413902282715, 1.235897183418274, 1.1661121845245361, 1.204063057899475, -0.3483694791793823, 0.03419853374361992, 1.1411995887756348, -0.8078517913818359, -0.22939249873161316, -0.7377912998199463, -0.14028817415237427, -0.639607310295105, 0.04527400806546211, 0.8639341592788696, 0.25108233094215393, 2.1428356170654297, 0.7069922089576721, -1.4084080457687378, -0.363694965839386, -1.7309558391571045, 0.12606802582740784, 1.1821231842041016, -0.9323545694351196, -1.1231729984283447, -0.7557160258293152, -0.4328325092792511, -0.13682623207569122, 0.9328838586807251, 0.07702680677175522, 0.4106541872024536, 0.6233242750167847, 0.42583754658699036, 0.43563881516456604, -2.9229891300201416, -0.8722655177116394, 1.4479126930236816, 1.1043453216552734, 0.8903039693832397, 0.6809534430503845, -0.6672413945198059, -0.9782134890556335, 0.46894389390945435, 0.7842345237731934, 0.672682523727417, 1.9708714485168457, -1.9579246044158936, 1.5853760242462158, 0.24802489578723907, 1.150044322013855, 0.8432415127754211, 0.43506643176078796, -1.3031635284423828, -0.18824627995491028, -0.6939208507537842, -1.4916739463806152, 1.2327969074249268, 1.082241177558899, 0.8669576048851013, 0.25210151076316833, 0.10849028080701828, -0.9225547313690186, -0.23879168927669525, 0.6067783236503601, -1.2915838956832886, -0.5594595074653625, 0.11969731748104095, 1.270025372505188, -0.894654393196106, 0.26946431398391724, 0.4567091763019562, -0.26445066928863525, -0.6685287952423096, 0.8347055315971375, 0.16456282138824463, 1.2512624263763428, -0.19876031577587128, -1.225320816040039, 0.9216932058334351, -0.37516364455223083, -0.4099375903606415, -0.9024397134780884, -0.10144636780023575, 0.05441919341683388, 1.2008507251739502, -0.5244242548942566, 0.1806410551071167, -1.4664220809936523, 0.594418466091156, -0.8962509036064148, 0.5918450355529785, 1.4853646755218506, 1.5422357320785522, 0.8889707922935486, 0.5882193446159363, 1.739599585533142, 0.3985091745853424, 0.309776246547699, -0.34640729427337646, -1.1950708627700806, 1.6931298971176147, -1.1391712427139282, 1.0937517881393433, 1.0382671356201172, 1.4543274641036987, -1.0853763818740845, -0.7451505064964294, -0.9517815113067627, 0.0036346521228551865, 1.5906847715377808, -0.44039446115493774, 1.6632641553878784, 0.3161383867263794, 0.2324337214231491, 1.1346267461776733, -1.8257616758346558, -0.45817166566848755, 0.21875537931919098, 0.20513524115085602, -1.169663429260254, 0.8894641399383545, 1.6275261640548706, -0.8889955282211304, 1.8444405794143677, -0.1822608858346939, 0.6463983654975891, 0.7481332421302795, -0.885534405708313, -0.48703765869140625, 1.066767930984497, -0.13098329305648804, 0.06296514719724655, 2.0207724571228027, -0.7269228100776672, 0.32509157061576843, -1.6153461933135986, -0.4498146176338196, -0.4527919888496399, -0.9334061145782471, -1.258980393409729, -0.10857284814119339, -0.06106237694621086, -0.5927019715309143, 0.3192821741104126, 0.3320123851299286, -2.0610270500183105, 0.4673842489719391, -0.5601584911346436, 0.7803482413291931, 1.1808985471725464, 0.9224815368652344, -0.5097128748893738, 0.5283738970756531, 0.3433613181114197, 1.1978726387023926, -0.23449194431304932, -1.0353354215621948, 0.1180189773440361, 0.17041920125484467, 1.2532305717468262, -0.9282779097557068, -1.0327706336975098, 0.9144959449768066, -0.2356981337070465, 0.22996649146080017, -1.2499914169311523, -0.54253751039505, 0.8482740521430969, -0.6852813363075256, -0.23893196880817413, -2.38798189163208, 0.8208975791931152, 0.7817046046257019, 0.28728994727134705, -0.16048431396484375, -0.9038814306259155, 0.9224110245704651, -0.322744220495224, -0.47694990038871765, 1.357182264328003, 1.1370805501937866, -0.6204124689102173, -0.7922446727752686, -0.4943723976612091, 1.407845377922058, 2.7369256019592285, -2.152759552001953, -0.025440523400902748, 0.23813100159168243, 0.6496119499206543, 0.5266447067260742, 0.03729243203997612, -0.09970411658287048, 0.9168473482131958, 1.400347113609314, 0.8956142663955688, 0.9511228203773499, 0.509384274482727, -0.29432013630867004, 1.0693703889846802, 0.6816296577453613, 1.5170978307724, 0.258823037147522, 0.1840045154094696, -0.3874865770339966, -0.6589629054069519, -0.43073973059654236, -0.7184334397315979, -0.2573278546333313, 0.1798151582479477, 0.666425347328186, -0.25344446301460266, 0.1162342056632042, -1.8499959707260132, 0.016359025612473488, -0.09650912135839462, 0.6831871867179871, 1.3149300813674927, 1.356175184249878, -0.2665824890136719, -0.45908334851264954, -0.7899580597877502, -0.7200247049331665, -0.3771440386772156, 1.3281779289245605, -0.4280617833137512, -1.4058235883712769, -0.9536814093589783, 0.17479486763477325, 1.0214414596557617, -0.4410623013973236, 0.1038314551115036, 0.36832016706466675, -0.6466501355171204, -1.168699026107788, -0.2199081927537918, 0.4452388882637024, 0.4521940350532532, -1.1281957626342773, 1.9871923923492432, -0.9267947673797607, 0.5984437465667725, -0.027290459722280502, -1.1029096841812134, 0.41470423340797424, -1.6063860654830933, 1.1649349927902222, 1.4833985567092896, 0.725636899471283, -0.35158202052116394, 0.25377774238586426, 0.933850884437561, -0.8269736170768738, 0.9524446725845337, 0.6608940958976746, 0.19267696142196655, 1.1440626382827759, -0.24347876012325287, 1.8149824142456055, -2.0374515056610107, 0.18245196342468262, 1.1167711019515991, 0.16495564579963684, -0.6402050852775574, 1.0511177778244019, -0.13535766303539276, -0.009198238141834736, 1.078970193862915, 0.2863290309906006, -0.6484182476997375, -0.5470820069313049, 1.4243884086608887, 0.9042354226112366, -2.2304179668426514, -0.57842618227005, 0.5821694731712341, -1.8889074325561523, 0.8234410285949707, -0.6311694383621216, -0.8659008145332336, 1.1061660051345825, -0.399233877658844, -2.7227842807769775, 0.2777135372161865, -0.7217227816581726, 0.14838114380836487, 0.6213492751121521, 0.28369826078414917, -0.12984366714954376, 0.2623463273048401, 1.5212830305099487, -1.2065726518630981, 2.1158735752105713, 0.14556705951690674, -0.20618778467178345, 0.4206498861312866, -1.1635560989379883, -0.5209391713142395, -0.37342703342437744, -1.1026359796524048, 2.0395359992980957, -0.9960988163948059, 1.0431430339813232, -0.04658836871385574, 1.1358708143234253, -0.6956062316894531, -0.24835583567619324, 0.1095474511384964, 2.083787679672241, 0.031455524265766144, -0.47866594791412354, -0.39216694235801697, 0.269552081823349, 0.19506241381168365, -0.8140247464179993, 1.264000415802002, 0.36802899837493896, -0.24306847155094147, -0.7437019348144531, 0.43654319643974304, -0.09977477788925171, -1.6119155883789062, -1.4998962879180908, -0.38251540064811707, -0.5183244347572327, 0.9734269380569458, 0.08142442256212234, -1.3026809692382812, -0.6865161657333374, -1.2553493976593018, -0.48379239439964294, -1.654706358909607, 0.1633264720439911, -0.8161133527755737, -0.15189655125141144, 0.22064779698848724, 1.4943374395370483, 0.21003054082393646, 0.15066476166248322, -0.10346110165119171, -1.1133900880813599, 0.4817831814289093, -0.9668567776679993, 1.4229947328567505, -0.9735040664672852, 0.9957134127616882, -0.5945626497268677], [1.453574299812317, 0.2849641442298889, 0.03486962616443634, -0.2329958975315094, -0.11587558686733246, 0.062138669192790985, -0.40508562326431274, 0.03258809447288513, -0.19875091314315796, 1.4429473876953125, 1.098591923713684, 1.5344951152801514, 0.15666265785694122, -0.6483907699584961, 1.4538739919662476, -1.0790810585021973, -0.5794857144355774, 1.0963596105575562, -0.49359023571014404, -0.7446824312210083, 0.3983847200870514, -1.3374910354614258, 0.8035861849784851, -0.1954096406698227, -0.030812937766313553, -1.2452207803726196, -0.8956443667411804, -0.6931883096694946, -0.4802362322807312, 0.022175991907715797, -1.2204725742340088, -0.3858029544353485, 0.46749234199523926, -0.4880193769931793, -0.9052223563194275, -1.2335975170135498, -0.8130238056182861, 1.374322772026062, -1.4045662879943848, -0.09642107039690018, 1.8962841033935547, 0.25091445446014404, 1.9607226848602295, -1.2318651676177979, 0.31534266471862793, 1.3175289630889893, 0.4434162974357605, -0.7351828813552856, -0.4532030522823334, 0.0002987996267620474, -0.1044691875576973, 0.6928901076316833, -0.4593985676765442, 0.37667298316955566, -0.04024805128574371, -0.10340636223554611, -0.66487056016922, -0.2738344967365265, -1.7961108684539795, 0.14932012557983398, -0.5649772882461548, -0.5552499294281006, 0.06390246748924255, 1.0582735538482666, -1.0394903421401978, -1.190947413444519, 1.1858243942260742, -0.3949790894985199, 1.5025622844696045, 1.642538070678711, 0.457278847694397, -0.12075550109148026, 1.0137298107147217, -0.33139002323150635, -1.769170880317688, -0.6155127286911011, 1.0426608324050903, 0.6181387901306152, 1.1084610223770142, 0.3002912700176239, -1.3774504661560059, -0.27923962473869324, 0.6849602460861206, 0.43903452157974243, 0.6050944328308105, -0.494415283203125, 1.5681638717651367, 0.3503507375717163, 0.6305018067359924, 0.6273865699768066, -0.3851107060909271, -0.6638494729995728, -2.2010650634765625, -0.8393514752388, -0.5847694873809814, 0.5527448058128357, -2.0878679752349854, 1.5822380781173706, -0.4501744210720062, 1.8736740350723267, -2.054967164993286, 0.503696084022522, -0.2506634294986725, 0.5195307731628418, -1.1909432411193848, 2.596997022628784, -0.09471090137958527, 0.06258798390626907, 0.6104530096054077, 0.08484712988138199, 0.7742814421653748, -0.403272420167923, -1.5639121532440186, 0.012642592191696167, -0.2394971400499344, 0.5408469438552856, -0.3961421549320221, -0.6113282442092896, -0.18025833368301392, 0.0546797551214695, 0.03521442040801048, 1.1273534297943115, 0.015085491351783276, -1.1525179147720337, 0.6443055868148804, 0.6022028923034668, -0.9926244020462036, -1.133010983467102, -0.28252866864204407, 2.613565683364868, -1.3560181856155396, -0.9051514863967896, 1.1987720727920532, 0.2364051640033722, -0.1688568890094757, 0.012604464776813984, 0.138379767537117, -0.42296841740608215, 1.049742341041565, 0.42061901092529297, 1.179525375366211, -0.6561562418937683, -0.49989891052246094, 0.3381137549877167, 1.4734575748443604, -0.6658474802970886, 0.12332365661859512, -0.6023473143577576, 0.1802944540977478, 0.17127808928489685, -0.23817822337150574, -0.206990048289299, 0.4168443977832794, -0.5908973813056946, 0.8083620071411133, -0.8733712434768677, -0.44916394352912903, 0.20229080319404602, 0.5894266963005066, 0.03556186705827713, -1.4107754230499268, -1.8143538236618042, 0.3322238624095917, 1.088523507118225, -0.09000540524721146, -0.3707006871700287, 0.3698514997959137, 1.8158066272735596, -0.005669876467436552, -1.1278494596481323, -0.691917896270752, 0.886837899684906, 1.1967195272445679, -0.3011694848537445, 0.18987751007080078, -1.3858098983764648, 2.3729944229125977, -2.1750543117523193, -1.9579845666885376, 1.3081104755401611, -0.5281206965446472, 0.258442223072052, -0.9347257018089294, -0.534461259841919, -0.13063503801822662, 0.5135780572891235, 1.4476323127746582, -0.8574487566947937, 0.14803414046764374, -0.11477141827344894, 0.33717238903045654, 0.7983619570732117, 1.7109884023666382, 0.16016526520252228, -0.9059552550315857, 0.6166196465492249, 1.4328172206878662, -0.462594211101532, -0.20261718332767487, 0.7755586504936218, -0.1142028421163559, -0.7800904512405396, -0.6729708313941956, -1.286226749420166, 1.291609764099121, -0.5870924592018127, -0.07507853955030441, 1.0253727436065674, -0.19557061791419983, 0.8503618836402893, -0.3817150890827179, 0.5762922763824463, 0.30195459723472595, -0.8824195265769958, 1.2540949583053589, -0.04902496188879013, -0.18803544342517853, -0.09928305447101593, -1.378832459449768, 0.9185441136360168, 0.24130848050117493, -1.0108283758163452, 1.4818788766860962, -0.2144836038351059, 1.3245086669921875, -2.264414072036743, -1.5621989965438843, 0.7874976396560669, 0.6746507287025452, -1.3563401699066162, 1.370765209197998, -0.431071937084198, 0.04134450480341911, -0.1592770665884018, 0.6959085464477539, 1.2005983591079712, 0.5761964321136475, 0.9037337303161621, 0.899978518486023, -0.8910757303237915, 0.3654782176017761, -1.5041345357894897, 1.1942425966262817, 0.6409668326377869, 0.742587685585022, 0.44533029198646545, -0.5559390783309937, 0.2347380369901657, 0.5528263449668884, -1.1007616519927979, 0.7704339623451233, -0.9419423937797546, -1.7930132150650024, -1.1223511695861816, 0.00592983141541481, 0.3540286123752594, 1.7925798892974854, -1.4524070024490356, 1.6211144924163818, 0.012692180462181568, -0.17386606335639954, -0.4887249767780304, 0.038910772651433945, 0.3393752872943878, -0.6870488524436951, 0.44598689675331116, 2.0748698711395264, 0.08182704448699951, -0.14903631806373596, -0.7042872309684753, -0.9862006902694702, -1.8725041151046753, -0.8334314227104187, -0.4897780120372772, -1.3537014722824097, -0.981675922870636, -1.4086556434631348, -1.197870135307312, 0.14977163076400757, 0.5484919548034668, 1.0832256078720093, -0.8285704851150513, -1.2862902879714966, 0.21384704113006592, -0.21447747945785522, 0.00709317484870553, -0.21711775660514832, 1.3534696102142334, -1.547177791595459, -0.4018678665161133, -0.6416518092155457, 0.4674276113510132, -0.9997676610946655, 0.8162024617195129, 0.26283934712409973, 0.4373965859413147, 0.26492074131965637, -0.5789985060691833, 0.773231029510498, 1.6722453832626343, 0.42411211133003235, 1.9702825546264648, -0.23801486194133759, -0.3436667323112488, 0.7404454350471497, 0.3917919993400574, -0.931736946105957, 2.0012786388397217, -0.6103817224502563, -0.3092880845069885, -0.2816551625728607, 1.8813587427139282, 2.144397020339966, 0.705095648765564, -0.09237008541822433, 0.7422449588775635, 0.7799311280250549, 0.7341623902320862, 0.10209208726882935, 0.42473188042640686, 0.06386995315551758, 0.40430164337158203, -0.30188262462615967, 0.08721180260181427, -0.2742323577404022, 1.5585697889328003, -0.49687930941581726, 1.0768953561782837, 0.5586690306663513, -0.1569204032421112, -0.19212177395820618, 0.4185704290866852, 0.9862276911735535, 1.0481778383255005, 0.6479208469390869, 0.4247826337814331, 1.2317814826965332, -1.1357951164245605, 0.285571426153183, -1.2900683879852295, -0.7164091467857361, -0.23252956569194794, -0.3602978587150574, 1.9719070196151733, 0.905180811882019, 0.294735848903656, -0.2533382475376129, -0.4205349087715149, -0.7097118496894836, -0.26071441173553467, 0.6051638126373291, -0.974588930606842, -2.0441479682922363, -1.8353908061981201, 1.6271332502365112, 1.0494842529296875, -0.0913727879524231, 0.2894524931907654, 1.2162489891052246, 0.27383455634117126, -0.702940821647644, 0.9109996557235718, -1.0054993629455566, -0.18320712447166443, 1.0419483184814453, 1.4057567119598389, -0.13339866697788239, -0.10028257966041565, 0.12043028324842453, 0.3638714849948883, 0.2606409192085266, 0.8245911002159119, -0.4518314599990845, -0.5071114301681519, 1.377767562866211, -0.39521124958992004, -0.42619001865386963, 0.714514970779419, -0.19290976226329803, -0.9842187762260437, 0.8905156850814819, 0.46120744943618774, 0.6745420694351196, -0.7950233221054077, -0.343315452337265, 1.347768783569336, 0.09923692047595978, 1.248399257659912, -0.6651890873908997, -0.3118358552455902, -1.3701424598693848, -0.3197857141494751, -0.6852363348007202, -0.7936383485794067, 0.15656568109989166, 0.2139604389667511, 2.436408042907715, -1.137906789779663, -1.0745688676834106, 2.0844733715057373, -0.037553828209638596, 0.5583580732345581, 0.7377129793167114, 2.536027431488037, 0.5670573115348816, -0.5882816910743713, -1.2607640027999878, 0.33334246277809143, -1.3401727676391602, -1.342345118522644, -0.5778861045837402, 0.4934122860431671, 0.05433670058846474, 0.5613406896591187, -1.1198806762695312, -1.756555438041687, -0.27940258383750916, 0.036353349685668945, 0.1589190512895584, 1.286144733428955, 1.5276825428009033, -1.5618927478790283, -0.2746232748031616, 0.8071860074996948, 0.3831990361213684, 0.5629943013191223, 0.603140115737915, -0.9219725728034973, -0.2442246377468109, 0.1657729297876358, -1.8106598854064941, -0.4584077298641205, -0.08312629163265228, -0.16988521814346313, 2.426948070526123, 0.03929860144853592, -0.05834048613905907, 1.5908052921295166, 1.381186604499817, -0.5147081017494202, 0.19963181018829346, -0.4953058660030365, -0.4067259132862091, -0.06702979654073715, 1.0950746536254883, 0.2699856758117676, -0.008043363690376282, 3.068174123764038, 0.939057469367981, -0.8697761297225952, 1.2941383123397827, -0.14968553185462952, 0.17637552320957184, -1.1597137451171875, 0.13983026146888733, 1.3521662950515747, -0.3802012503147125, -0.9122562408447266, 0.9920702576637268, 0.2523225247859955, 0.16842490434646606, -0.27772173285484314, -0.2920577824115753, -0.5412695407867432, 0.5662739276885986, 0.45641615986824036, -0.4519612789154053, 0.7244811058044434, 0.37596675753593445, -0.6283131241798401, 1.565251350402832, 0.34007617831230164, 1.6204849481582642, -0.5203521847724915, 0.13652406632900238, 0.5258036851882935, 1.9722079038619995, 1.622453212738037, 0.1758398860692978, 0.5174922347068787, -2.114823341369629, 0.5777416825294495, 0.6398260593414307, 1.934259295463562, -0.5845012664794922, -0.5082293748855591, -0.6714810132980347, -0.0039468612521886826, -0.5208147764205933, -1.3479546308517456, -1.0758188962936401, -0.5966097712516785, -1.2229772806167603, 1.1659245491027832, -0.8158668279647827, -1.996943473815918, 0.928493082523346, -0.7698847651481628, 0.527497410774231, -2.1295483112335205, -1.3361985683441162, 0.5359331369400024, -0.16462169587612152, 1.831588625907898, -0.38744956254959106, 0.37838298082351685, -0.5172494649887085, 1.343532919883728, 1.2045135498046875, -0.8717762231826782, 0.20248237252235413, 0.17300890386104584, 0.7761308550834656, 0.38034507632255554, 0.12147524207830429, -1.6363494396209717, -1.0224175453186035, -0.1992880254983902, 2.9835991859436035, -0.8990952372550964, 0.4114445149898529, -3.30399489402771, -0.9037792086601257, -1.8145644664764404, -1.1769347190856934, -2.4867823123931885, 0.48242104053497314, 1.842543363571167, 0.44089993834495544, 0.17148788273334503, 0.319510281085968, -0.5421441197395325, 0.27265825867652893, -1.0772393941879272, -2.2938144207000732, -0.7799589037895203, 0.9295192956924438, 0.7987397909164429, 0.5683183073997498, -0.6030682325363159, -1.2951982021331787, 0.08529786765575409, -0.9660999774932861, 1.3121018409729004, -0.4032772183418274, 1.2951104640960693, 0.7685366272926331, -1.6914794445037842, -0.47939416766166687, 1.0446358919143677, -0.5306496024131775, 1.0205421447753906, 1.1478562355041504, 1.1761611700057983, -0.5419095158576965, 1.4517768621444702, 1.2145798206329346, 0.7780675292015076, -0.4559459388256073, 0.05510983616113663, 0.5551190972328186, 0.4846770763397217, -0.6304397583007812, 0.4745080769062042, -0.5953343510627747, -1.2661887407302856, 1.214136004447937, 0.39026156067848206, 1.865214467048645, 0.05532219633460045, -1.0294638872146606, 0.2664279043674469, 1.5106217861175537, 0.25464779138565063, 0.6658299565315247, 1.5550509691238403, 0.28865087032318115, -0.0018187895184382796, -1.6190744638442993, 0.17307531833648682, 0.37421664595603943, -0.11247630417346954, 0.10078620165586472, 0.3817109763622284, -0.07840082794427872, 0.495452344417572, 1.0026681423187256, 0.23731659352779388, 0.7736101746559143, 0.4704996347427368, -0.4338514804840088, -0.49966493248939514, -0.7329779267311096, 0.45829740166664124, 0.988802433013916, -1.8845216035842896, -1.317905306816101, -1.3092811107635498, -2.1877381801605225, -1.4361051321029663, -0.9468162655830383, -0.2673150300979614, -0.5810073614120483, 1.008863091468811, 0.5407407283782959, -0.36436885595321655, 0.2991672456264496, -1.1512681245803833, -0.2948158085346222, -0.7792778611183167, 1.013920545578003, -0.5311822295188904, 0.9906898736953735, -0.33695152401924133, 0.4036213755607605, -2.1324474811553955, 0.03025531768798828, 0.07558562606573105, 1.6371498107910156, 0.25068268179893494, 0.2754057049751282, -0.8337182998657227, -1.0461796522140503, -0.8033438920974731, -0.06804607808589935, -0.6699101328849792, 0.6002383828163147, -0.5152201652526855, -0.8302851915359497, -0.6721357703208923, -0.6978859901428223, -0.45101678371429443, 0.47225454449653625, -1.0924344062805176, -0.428043395280838, 0.09770167618989944, 0.3416740298271179, 0.2129797637462616, -0.2402830719947815, 0.481128990650177, 0.7963300943374634, 0.09525205194950104, 0.5609914660453796, 2.2969424724578857, -1.0120257139205933, -0.08133425563573837, 0.09099315851926804, 0.6610391139984131, -0.9464514255523682, 0.5742371678352356, 0.19794516265392303, 0.20186494290828705, -0.567813515663147, -1.1929479837417603, 0.8712278008460999, -0.5082846283912659, 0.5564976930618286, -0.1126413345336914, -0.4215732514858246, 0.7769769430160522, 2.113334894180298, 0.42100346088409424, -0.4121268689632416, 1.0536961555480957, 0.13268785178661346, -0.1930616796016693, -0.6230986714363098, 0.5979463458061218, -0.9667851328849792, -0.18197216093540192, -0.26000818610191345, 1.3474746942520142, 1.212761402130127, -0.44703903794288635, -0.02646317146718502, -0.33624497056007385, 1.2322856187820435, 0.05757889896631241, -1.7939420938491821, -0.3613077700138092, -0.33475637435913086, 0.7818862795829773, 0.7669283151626587, -0.7074206471443176, -1.5573776960372925, 0.009364884346723557, 1.3959541320800781, 0.1285935938358307, 0.28120243549346924, 0.6542072296142578, 0.8466693758964539, -0.22397693991661072, -0.8425620794296265, 0.5775407552719116, 0.10901359468698502, 1.1597964763641357, -0.4085950255393982, -0.041631679981946945, 0.22136656939983368, 0.05117632821202278, -1.0149469375610352, -0.12458936125040054, 1.1339702606201172, 0.4570496678352356, 0.3994707465171814, -0.378248929977417, -1.8989121913909912, 0.40939390659332275, 1.2094695568084717, -0.9152504205703735, -0.6368840932846069, 1.0114738941192627, 0.01935877837240696, 0.6332361102104187, -0.4505530893802643, 0.44222894310951233, -1.6135547161102295, 2.7114155292510986, -0.587106466293335, -0.7223053574562073, 0.6286138892173767, -0.48393887281417847, 0.196611225605011, 0.25876128673553467, 1.3986173868179321, -0.21765880286693573, 0.8864459991455078, -0.3023029863834381, 0.09241313487291336, -0.4553017020225525, 0.2557200491428375, 1.1485754251480103, -0.009656978771090508, -1.6226108074188232, 1.372359037399292, 1.292781114578247, 0.9189838767051697, 0.09386185556650162, 0.13318733870983124, 0.998795747756958, -2.052297353744507, 1.1089417934417725, -0.9949661493301392, -1.5137656927108765, -2.3882229328155518, -1.9274457693099976, -0.06863993406295776, -1.6728637218475342, 0.8046307563781738, -1.856275200843811, -0.5436803102493286, 0.15350598096847534, -0.4595102071762085, -0.737058699131012, 2.5358850955963135, 0.9307140707969666, 1.012315034866333, 0.19533982872962952, 0.6930926442146301, 0.3517582416534424, 0.750961184501648, -1.3391362428665161, 1.6412090063095093, -0.055838800966739655, -0.5469015836715698, 1.1557023525238037, 1.1431262493133545, 1.048477292060852, 0.3271165192127228, -0.41488784551620483, -1.1536320447921753, 0.36716654896736145, 0.7517780065536499, 0.8659553527832031, -0.9970313906669617, -0.7756751179695129, -1.3088432550430298, 0.08582046627998352, 0.6720019578933716, 1.5755674839019775, 1.9585552215576172, -0.3815244436264038, 1.4080743789672852, 2.215634822845459, -2.3266987800598145, -0.860441267490387, -0.40754950046539307, 0.2032371610403061, -0.0725393146276474, -0.9186020493507385, -0.5789574384689331, -1.1834839582443237, -0.1716567724943161, -0.3904797434806824, 0.37510916590690613, 0.046278707683086395, 0.0016175172058865428, -0.813876748085022, 0.8352579474449158, -0.26669707894325256, 1.1701833009719849, -0.8185968995094299, -0.20661647617816925, -0.5567295551300049, -0.07084812223911285, -0.35664138197898865, 0.050599873065948486, -0.28294211626052856, -0.018305236473679543, 1.5711721181869507, 0.5847514867782593, -1.3238903284072876, 0.08494057506322861, -0.6090601682662964, -0.35924166440963745, -0.6245471239089966, 0.7413740158081055, 0.5353318452835083, -0.26841801404953003, -0.3080103397369385, -0.932769238948822, -1.4862005710601807, 2.4221668243408203, -0.30651333928108215, -0.4138055741786957, 0.5233486890792847, -1.2867320775985718, -2.081707715988159, -0.3660542666912079, -0.2936251759529114, 1.2864607572555542, -0.33334243297576904, 0.6575204133987427, -1.2640380859375, 0.2601710855960846, 0.7068317532539368, 1.3211885690689087, 1.9060553312301636, 1.394545078277588, 0.5274787545204163, 0.8224354386329651, -0.11684025824069977, -1.2859212160110474, 3.4317257404327393, 2.3190970420837402, 0.38970181345939636, -1.3537534475326538, 0.6792728304862976, -0.7121499180793762, 1.8941256999969482, -2.2109286785125732, 0.872246265411377, 1.322893500328064, 1.569408893585205, 0.08036737889051437, -0.45004788041114807, -0.9344698786735535, 0.5328422784805298, 1.4051401615142822, 2.0856668949127197, 2.042375087738037, -1.5080888271331787, -0.5501425862312317, -0.02208259515464306, -0.6201429963111877, -2.231539487838745, -0.3772425949573517, 1.5281118154525757, -0.4478307366371155, 0.5604853630065918, 0.28942641615867615, 0.1961289346218109, 1.5913639068603516, -0.05939822271466255, -0.49703943729400635, -0.8315731287002563, 0.33556094765663147, 0.5083456039428711, -1.6312919855117798, -0.675312876701355, -0.38150957226753235, -0.783251941204071, 0.8413758277893066, 0.3815596103668213, 1.655953288078308, 0.24480101466178894, -1.0439432859420776, -1.2299355268478394, 0.20640990138053894, -1.1319211721420288, 1.3068960905075073, -0.6011291742324829, -0.6787518858909607, 0.44084039330482483, 1.079923152923584, 1.7107752561569214, -0.8579082489013672, 0.5746026039123535, -0.905989944934845, 1.758309006690979, -1.1551756858825684, -0.7112039923667908, 0.3338511884212494, 0.6166605353355408, -1.1622666120529175, -0.10872465372085571, 1.0158065557479858, -0.6545487642288208, 1.387107014656067, 0.5483084917068481, 0.2927626967430115, 0.2191183716058731, 2.203573703765869, 0.5839414596557617, -0.5518473982810974, -0.4706977903842926, -0.007939315401017666, 0.09671761840581894, -0.8915483951568604, 0.7324965596199036, -0.5482462644577026, 0.8162911534309387, -1.9162797927856445, 1.7151873111724854, -0.24725133180618286, 0.9902399182319641, 1.582628846168518, 0.5688777565956116, 1.4028520584106445, 0.1420530527830124, 0.7165292501449585, 0.45566579699516296, -0.47868338227272034, 1.4535653591156006, -2.344167947769165, 0.33910131454467773, -0.2237132042646408, -2.055833578109741, 0.8612016439437866, -0.5367803573608398, -0.5981598496437073, -0.09315921366214752, -0.11588650196790695, -0.6677559614181519, 0.9966772794723511, -1.0494569540023804, 0.23596593737602234, -1.1516995429992676, 0.8086206316947937, 1.8200037479400635, 0.25614553689956665, -0.6655607223510742, 0.11130780726671219, -0.0028778270352631807, -0.6383351683616638, 0.04918279871344566, 0.8548195958137512, -0.6324046850204468, 1.310054063796997, 0.7418535947799683, -0.5057663917541504, -0.4268510937690735, -0.7872117757797241, 0.39674848318099976, -2.1774024963378906, 0.01505456492304802, -0.7017079591751099, 1.2669298648834229, -0.23467722535133362, -0.2987625002861023, -2.557631731033325, -0.25796547532081604, -0.3335108160972595, 1.2735321521759033, 0.5046687126159668, -0.5638898015022278, -0.458323210477829, 1.9095436334609985, 0.20371238887310028, 1.2267849445343018, 0.4129612147808075, 1.022240400314331, -1.3645384311676025, -0.9740864038467407, -1.976797342300415, 0.4808177053928375, -0.46227359771728516, -0.12746642529964447, 1.1881670951843262, -0.8142697215080261, -1.3590034246444702, 1.5499672889709473, -0.8002285361289978, 0.40468403697013855, -1.2583414316177368, 1.2498880624771118, -1.5580662488937378, 1.4952661991119385, -1.001043677330017, 0.5139548778533936, 0.8867759704589844, 1.0442012548446655, -0.127449631690979, 0.3558294177055359, 0.07269106805324554, 0.17587821185588837, 0.28257906436920166, 0.21323879063129425, 2.294309616088867, 1.961069107055664, -1.621656894683838, -0.4176592528820038, -0.629466712474823, 1.9156757593154907, 0.18525376915931702, 0.29027825593948364, 1.3455241918563843, -0.17209360003471375, -0.4626055359840393, 0.19441112875938416, 1.3625667095184326, -0.59932941198349, -0.7502816915512085, 0.6335740089416504], [-1.2774739265441895, -0.646263599395752, -1.4142004251480103, 1.1178916692733765, -1.1988297700881958, 1.2478629350662231, 0.6264635920524597, 0.2440662533044815, 0.28321588039398193, -0.6432669758796692, 0.33814024925231934, -0.4981127381324768, -1.3320878744125366, -0.39075690507888794, 1.3193025588989258, -0.47512853145599365, 0.3525245487689972, 1.9964979887008667, -0.22007273137569427, 0.20476675033569336, -0.2180783748626709, 1.682626724243164, 0.04760287329554558, 1.7559545040130615, -0.48170235753059387, -0.6011133790016174, 1.0061006546020508, -0.8355271816253662, -1.7313812971115112, 0.4565528929233551, -0.46780845522880554, 0.030169347301125526, 0.17690525949001312, -0.46158331632614136, -1.2866439819335938, 0.5985701084136963, 0.293154239654541, -0.9057167768478394, -0.992241621017456, -0.5966012477874756, -0.3980604112148285, -0.4584379494190216, 0.6982299089431763, 1.0792044401168823, -0.551474928855896, -1.000205397605896, -0.7268849015235901, 0.5498055815696716, -1.482904314994812, -0.5168947577476501, 1.3491398096084595, 0.09114614129066467, 0.3704245984554291, -1.1670044660568237, 2.0651941299438477, 0.6971467733383179, 0.20887801051139832, -2.734246253967285, 0.955026388168335, -0.1824926733970642, -0.5797996520996094, 1.3122385740280151, 0.31852853298187256, -1.5037857294082642, 0.370123952627182, 0.8839236497879028, -1.1404763460159302, 1.1074678897857666, -0.09370851516723633, 0.3473072946071625, 1.1620912551879883, 1.4279636144638062, 0.2555481493473053, -1.2912052869796753, -0.13025325536727905, -0.8833592534065247, 0.5298587083816528, -1.5953611135482788, -0.14763082563877106, 0.2997745871543884, -0.7549254298210144, -1.2540903091430664, -1.0133789777755737, 1.008396029472351, 1.9901944398880005, 0.05512735992670059, 0.6002488732337952, -0.6534246206283569, -0.21529032289981842, -1.3136101961135864, 0.5590601563453674, -0.6805261969566345, -0.5798685550689697, -0.23759186267852783, -0.3679291009902954, -1.4851231575012207, -2.2309279441833496, -0.3860146999359131, -1.18068528175354, -0.2698368728160858, -0.6572232246398926, -0.7002363801002502, -0.6453568339347839, -0.7512403726577759, -0.7635264992713928, 0.827512264251709, 1.080715537071228, -0.36984267830848694, -0.89789217710495, -1.65330171585083, 0.32200807332992554, 1.1387296915054321, -0.8619294762611389, -0.8150925636291504, 0.5834419131278992, -0.01457370724529028, 0.15607541799545288, 0.5795299410820007, -1.5589267015457153, -0.42741385102272034, -0.052264947444200516, -0.2876541018486023, 0.007088702637702227, -0.31213659048080444, 0.7179572582244873, 1.6640515327453613, 0.48848411440849304, -0.7694827914237976, 0.9682227969169617, -1.069368839263916, 0.04692316800355911, -0.13057465851306915, 0.1986771523952484, -0.44347766041755676, 0.24139097332954407, 0.5902924537658691, -1.2252516746520996, 1.2596336603164673, -0.7159164547920227, 2.0402982234954834, 0.6612810492515564, 0.6340022087097168, 0.3789956569671631, -0.7457254528999329, -0.2558424174785614, 0.21163636445999146, -1.9942331314086914, 2.2628421783447266, -0.3839487135410309, 0.18024864792823792, -0.13922613859176636, -1.2630826234817505, -1.8779640197753906, -1.3802410364151, 0.7356938719749451, -0.7121272087097168, 1.8206088542938232, 0.22158269584178925, 0.8818023800849915, 0.1543913185596466, 0.8548917770385742, -0.27264100313186646, -0.40927600860595703, 1.776140570640564, -0.7499189376831055, -1.6319228410720825, 0.5730546712875366, -0.22795410454273224, 0.9861419796943665, -1.4381678104400635, 0.29464447498321533, 0.8507421016693115, 1.06298828125, -0.06499230116605759, -1.9643126726150513, -0.17174361646175385, -0.026655355468392372, -0.8701410889625549, 0.6856902241706848, -0.12560757994651794, 0.7887510061264038, -0.1836097091436386, 2.9914746284484863, -0.40714356303215027, -1.3326743841171265, 2.547921895980835, -1.597355842590332, -0.10325408726930618, 0.5019246339797974, -0.5526784658432007, 1.128656029701233, 0.6964516639709473, -0.3829536736011505, 3.0295512676239014, -0.08101838827133179, -0.41567206382751465, -1.3175755739212036, 0.5442758798599243, -0.5070121884346008, 0.37757736444473267, 0.5113762021064758, 1.166942834854126, 0.20645645260810852, 0.7671040892601013, -1.4208984375, -0.352895587682724, -0.6612955927848816, 0.8095126748085022, -0.12857434153556824, -0.42358437180519104, 0.0823850929737091, -0.15175291895866394, 0.4529258608818054, -0.5886201858520508, 0.6008101105690002, -0.14791102707386017, -0.5417410731315613, -0.6640807390213013, 0.512911856174469, 1.4058129787445068, 0.5877540707588196, -2.5083677768707275, 1.3322815895080566, 0.05861157923936844, 0.9021425843238831, -0.5366132855415344, 0.687881350517273, 0.5962181091308594, 0.3564777970314026, 0.6729868054389954, 0.32618269324302673, 0.5563559532165527, 0.8568461537361145, 1.0243111848831177, -1.1353589296340942, -0.08761540800333023, -1.6896655559539795, 1.1888272762298584, -0.2835560739040375, 0.316302090883255, 1.1145519018173218, -0.05595981329679489, 0.5521153211593628, -0.5743149518966675, -1.3862833976745605, 1.682225227355957, 1.215047001838684, 0.4595467448234558, -0.21935048699378967, 0.018970826640725136, -1.6825084686279297, 0.5460894107818604, 0.4533887207508087, -2.4836297035217285, -1.6857298612594604, 1.938965916633606, -0.6495144367218018, 0.2742345333099365, 2.178887367248535, 1.3086620569229126, 0.2193261682987213, 0.8589466214179993, 0.40678614377975464, 0.644484281539917, -0.38467201590538025, -0.23492495715618134, 0.09617990255355835, -0.18696238100528717, -0.8700498938560486, 1.793630838394165, 0.4109392464160919, 0.5200165510177612, -0.6588167548179626, 0.8086045384407043, -0.3897938132286072, -0.6311765313148499, -0.743747353553772, 0.1701502799987793, -1.9578787088394165, 0.7256683707237244, -1.2703189849853516, 1.6379281282424927, 0.023964503780007362, 1.0586012601852417, 1.237377643585205, -1.135046124458313, -0.08048377186059952, 0.9746570587158203, -0.45093628764152527, -1.237810492515564, 0.9852277636528015, -0.4743247330188751, -1.1007527112960815, -0.6359646916389465, -0.7036853432655334, 0.20140038430690765, 1.4854944944381714, -2.125910997390747, 0.5183559656143188, -0.3769915997982025, -0.03319430723786354, 0.9424680471420288, -0.9971190690994263, -0.04242650419473648, 1.1055248975753784, 2.4753262996673584, 1.126450538635254, -0.3805182874202728, 0.5145000219345093, 1.6013675928115845, 0.8278875946998596, 0.3056153655052185, -0.7368233799934387, -0.7966682314872742, -0.5368676781654358, 1.1668237447738647, -1.1405857801437378, 0.3941454589366913, 0.29138556122779846, 0.2543025016784668, -0.4037754237651825, -1.6874088048934937, -1.4526631832122803, -1.5098037719726562, 0.18871694803237915, -0.5381036996841431, -0.2197141796350479, 0.4146726429462433, 0.2984590232372284, 1.6581363677978516, -0.7024618983268738, -1.4175113439559937, 0.5720210075378418, -0.5316453576087952, -0.474209189414978, 0.6012077331542969, -0.14593902230262756, -0.6961994767189026, -0.3934043049812317, -0.4333930015563965, 2.3041162490844727, 1.9528346061706543, 0.37880939245224, -2.5422167778015137, 0.007077877875417471, -0.569424033164978, -1.5896157026290894, -0.42159411311149597, 1.0052717924118042, 0.1491115391254425, -0.7621734738349915, -1.5665414333343506, 0.015533692203462124, -0.2563999593257904, -1.4463858604431152, 0.8152368068695068, -0.5601900219917297, -0.43763455748558044, -1.1547712087631226, 1.1241506338119507, 0.5168502330780029, 1.2777245044708252, -1.013574481010437, 0.5344777703285217, 2.132920026779175, 0.3812238276004791, -0.5506913065910339, 0.2645311653614044, 0.1748402714729309, 0.2920216917991638, 1.1474970579147339, 0.42945006489753723, -0.8236391544342041, -0.8070153594017029, 1.3325132131576538, -0.045733965933322906, -0.1488780528306961, 1.068200707435608, 1.853739857673645, -0.6777093410491943, -1.136032223701477, -1.5042366981506348, 0.6255589127540588, 0.06217123195528984, 1.2277027368545532, -0.11262621730566025, -0.9070389866828918, 0.5823394656181335, -0.14996495842933655, 0.22393618524074554, 1.2439961433410645, -0.011102413758635521, -0.7746902704238892, 0.6406069993972778, 0.30975982546806335, -0.2717553973197937, 0.3807556927204132, -0.16981160640716553, -0.5271902084350586, -0.1584729701280594, 1.0241649150848389, -0.48580819368362427, 0.03158245608210564, -0.3338359594345093, -0.5008300542831421, -0.6316401362419128, 0.5107417106628418, -0.34326234459877014, -0.49859121441841125, -1.2432807683944702, 0.03932555764913559, -0.41649457812309265, 1.2858906984329224, 0.5077890157699585, 0.45034199953079224, -0.4648468494415283, -1.923458218574524, -1.421524167060852, 0.24952153861522675, -0.05372193083167076, 0.8531038761138916, -1.0760890245437622, -0.4437514543533325, 0.31461748480796814, -0.07521680742502213, 0.056905701756477356, -0.3696044087409973, 0.15211112797260284, 0.8819385766983032, 0.7591219544410706, 0.6872996687889099, -2.4271974563598633, 0.9689595103263855, -1.0554536581039429, -0.6620627045631409, -0.7141486406326294, -0.4297647178173065, 1.5848997831344604, -1.7548080682754517, 0.15810954570770264, -2.296293020248413, 0.3428764045238495, -0.6561592221260071, -1.3430466651916504, -1.5860127210617065, -0.39627352356910706, -0.8852158188819885, 0.648122251033783, -0.225555881857872, -0.21068553626537323, -0.6331736445426941, 0.26181119680404663, 0.24543415009975433, 0.5893924832344055, 2.4165403842926025, -0.883629322052002, 0.034434232860803604, 0.5990718007087708, -0.369721919298172, -1.9208779335021973, -2.202826499938965, -2.292933940887451, 1.379391074180603, 0.6361377239227295, 1.2957983016967773, -1.0946763753890991, -0.6681604385375977, 1.9655194282531738, -0.18792980909347534, 0.09584614634513855, -0.44174090027809143, 1.4506314992904663, -1.3874149322509766, 0.6680596470832825, -0.7409877181053162, -1.023311734199524, 0.024873200803995132, 0.7569490075111389, -1.0464625358581543, -0.052972469478845596, -0.8257443904876709, -1.71031653881073, -1.2112128734588623, 0.07643184065818787, -1.7427608966827393, 1.1497219800949097, -0.47462722659111023, -1.2595950365066528, 2.3563365936279297, 1.102391242980957, 0.08427325636148453, 0.31900331377983093, 0.007980824448168278, -0.13252176344394684, 0.2131120264530182, -0.48835235834121704, -1.3913689851760864, 0.8336525559425354, -0.9965165257453918, 0.3993244469165802, -1.61373770236969, 0.41986751556396484, 0.23216556012630463, -0.4291703999042511, -0.6044743657112122, -0.17732109129428864, 0.4035646319389343, 0.6505740284919739, -0.6937594413757324, -1.5172078609466553, 0.20443852245807648, -0.8091909885406494, -1.6095842123031616, 1.3574098348617554, -0.8134990930557251, -0.29318517446517944, 0.12023335695266724, -0.8435951471328735, -0.7373397946357727, -0.3167484402656555, -0.06466492265462875, 0.1417783945798874, 0.2895174026489258, -1.2247191667556763, 0.31024062633514404, 1.0157108306884766, 1.0256010293960571, -0.005283643491566181, 0.6021291017532349, 0.32916393876075745, 0.20656727254390717, 0.6779862642288208, 1.5492304563522339, 0.898064911365509, 1.597741723060608, 0.5468477606773376, 2.5802669525146484, 1.608039140701294, -1.610888123512268, -0.9148206114768982, 0.702331006526947, -1.9052705764770508, 0.6822024583816528, -1.9343135356903076, 0.27098506689071655, 0.05652299523353577, 0.20808537304401398, -0.4006164073944092, -0.2526448965072632, -0.05068865045905113, 1.482688069343567, -0.9992818236351013, 1.820571780204773, 0.7460052371025085, -0.23025386035442352, -2.6319832801818848, 0.3436248302459717, 0.6605004668235779, -0.8813959360122681, -1.3909006118774414, 0.5112444758415222, 1.7277469635009766, 1.1918138265609741, -0.4151875972747803, 1.5661492347717285, 0.2438362091779709, -1.1069331169128418, -2.5144803524017334, 1.6609809398651123, 0.20953528583049774, 1.0282789468765259, -2.5682761669158936, -0.5920019745826721, 1.2064427137374878, -0.2780514061450958, -1.1330734491348267, 0.025510158389806747, -1.1360913515090942, 0.8682432174682617, 1.1370265483856201, 1.6148093938827515, 0.2431236058473587, 0.5620161294937134, 0.46232813596725464, 1.3238688707351685, -0.21102012693881989, -0.6104227304458618, -0.5052590370178223, 0.44187524914741516, 0.1433841437101364, 1.8259451389312744, -0.21119703352451324, -0.5661852955818176, -2.113875389099121, 1.1291018724441528, 0.6291171312332153, -0.8176189064979553, -1.6657121181488037, -1.6882600784301758, 2.4065744876861572, -1.3323298692703247, 1.6549193859100342, -1.0793992280960083, -1.5978634357452393, 0.14745394885540009, 1.1296017169952393, -0.4340701699256897, -0.6734240651130676, 0.8607616424560547, -0.6905761957168579, -0.8566325306892395, 0.17409132421016693, 0.06532592326402664, -0.17052997648715973, 0.2695085108280182, -0.3999415636062622, -0.40233123302459717, -0.2618689239025116, -1.019249677658081, -0.15576209127902985, -0.3535376787185669, 1.967105507850647, 1.6360105276107788, -0.4118868112564087, -1.6290076971054077, 0.09270042181015015, -0.3550073206424713, -0.5987988710403442, -1.419669270515442, 1.4634556770324707, -0.7594993710517883, -1.769161581993103, 0.5172415375709534, -0.11217004805803299, 0.17622564733028412, -1.1208019256591797, 0.4314495027065277, -1.014882206916809, -0.9728641510009766, 1.2935718297958374, -0.5522692203521729, -0.05301060155034065, 1.64044189453125, 0.6719211339950562, -1.7457009553909302, -0.36365780234336853, -0.1921861618757248, -0.12312375754117966, 0.3474709391593933, 0.7725865244865417, 0.031741660088300705, 0.5185686349868774, -0.5799873471260071, -0.11876876652240753, -0.08385422825813293, 0.541994035243988, 0.03612585365772247, 2.301969289779663, 1.372764229774475, -0.8262702226638794, -0.601188600063324, 0.1412326842546463, 0.9457338452339172, -2.076883554458618, -1.6853585243225098, 0.3835652768611908, 0.7026429176330566, -1.0907115936279297, -0.9336318969726562, 2.1758437156677246, 1.4567499160766602, -0.5343855619430542, -1.8914597034454346, -0.3222411870956421, 0.3964758515357971, -0.42023470997810364, 0.3141714930534363, -0.1268082708120346, 2.513063669204712, -0.22327126562595367, 0.14027941226959229, -0.9817735552787781, 0.2011842429637909, -0.14469148218631744, 1.0764410495758057, -0.3886334002017975, -0.9071871042251587, 0.3688892722129822, -0.09998509287834167, -1.2191448211669922, -0.17735262215137482, -0.15360911190509796, -0.5280447006225586, -0.6010277271270752, -0.33800628781318665, 0.25157421827316284, 0.9730634689331055, -0.7328510880470276, 0.15826921164989471, 0.2637246549129486, 1.4880291223526, 1.3005634546279907, -0.31028854846954346, 0.18114280700683594, -0.5630052089691162, -1.8321114778518677, 1.0024579763412476, -0.6706135869026184, -0.49793803691864014, 0.3369941711425781, 1.5751100778579712, 0.8449237942695618, 0.4222821295261383, -0.0981694832444191, -1.1224766969680786, 0.35015854239463806, -1.8687958717346191, -0.08351121842861176, 1.128222942352295, -0.9511712789535522, 0.920011043548584, 1.3941715955734253, 0.9785273671150208, -0.46969857811927795, -1.0261619091033936, 1.2384321689605713, -1.1399686336517334, -0.8658154606819153, 1.3181897401809692, -0.9462779760360718, -1.5488601922988892, 0.694034993648529, -1.3805829286575317, -0.1302051544189453, -0.5088337063789368, -2.3101141452789307, -1.807721495628357, 1.5126909017562866, -0.13052217662334442, -0.8654792308807373, -0.24490445852279663, 0.4633559584617615, -2.3914482593536377, 0.7907868027687073, 0.9585703611373901, 0.04185252636671066, -1.6185001134872437, -0.47232502698898315, -0.6203987002372742, -0.6993441581726074, 1.9714065790176392, 0.9600573182106018, -1.6348134279251099, -0.25017204880714417, 1.8932396173477173, -0.3586932122707367, 0.33485618233680725, -1.1101313829421997, 0.05586822330951691, 2.38315486907959, -1.2305312156677246, 1.2146906852722168, -1.2915430068969727, 0.24778670072555542, -0.21924647688865662, 0.8149267435073853, -3.8274123668670654, 1.5135167837142944, 0.6326014995574951, -1.7915468215942383, -0.11663128435611725, -0.49247679114341736, -0.28850868344306946, 1.0855729579925537, -0.03306063264608383, -0.7965068817138672, 1.571872591972351, -0.26584795117378235, -1.3529020547866821, -1.7425395250320435, -1.1622902154922485, -1.529341697692871, 0.5491348505020142, -0.2470369040966034, 0.6793103218078613, 0.16550910472869873, 0.8097636103630066, -0.1573452353477478, -0.43926090002059937, -1.6476634740829468, -0.287215918302536, 0.8486451506614685, 1.0912199020385742, -2.3917906284332275, -0.7747898697853088, 0.07246585190296173, -1.4806512594223022, 0.8262995481491089, 0.8599639534950256, -2.7216145992279053, -0.3470200300216675, 1.0318197011947632, -1.2511132955551147, -0.3223661482334137, -0.780565619468689, 0.26009994745254517, 0.5330519080162048, 0.8827792406082153, 0.8956127166748047, 0.42412930727005005, -0.5520654916763306, 0.7231510877609253, -0.6923902630805969, 0.6932060718536377, -0.6894424557685852, 0.37711235880851746, 1.8996857404708862, 0.5766152739524841, 0.049026958644390106, -0.9571461081504822, 0.4433944523334503, -1.422359585762024, -1.6123979091644287, -0.010988239198923111, 1.2985199689865112, 0.5488438606262207, 0.6074957251548767, 0.9368982911109924, -2.000164747238159, -0.5305197834968567, -1.0390865802764893, -0.9233435392379761, -0.5222741961479187, 0.39058709144592285, 0.39740484952926636, -1.1893088817596436, 1.0171635150909424, -0.8782946467399597, -0.6496691107749939, 0.9238241314888, 1.1787675619125366, 1.6649322509765625, -0.17066827416419983, -1.000551462173462, -0.3361189067363739, 1.9222155809402466, 0.8927410244941711, 1.0712394714355469, 1.565049171447754, -0.135229229927063, 1.239243984222412, -0.16489602625370026, -1.0577200651168823, -0.6364248991012573, -1.2573343515396118, -0.09442911297082901, 0.43147286772727966, 0.03651156648993492, 1.551912784576416, 0.0584881454706192, -0.2224709689617157, 0.9135652184486389, 1.4898067712783813, 1.0876712799072266, 0.4635733366012573, 1.323254108428955, -0.761127233505249, 0.03324706852436066, 0.6592426896095276, 0.7154805064201355, 0.603210985660553, 0.16973812878131866, -0.6309612989425659, -1.0816584825515747, 0.9098930954933167, -2.0441091060638428, -1.484663724899292, 0.44628822803497314, -0.6275046467781067, 0.3178436756134033, -0.045647721737623215, -1.2931547164916992, 1.5486401319503784, 0.664103090763092, -0.8192796111106873, -0.38422736525535583, -0.3016723692417145, 1.0790414810180664, 1.3522207736968994, -0.4265204071998596, -1.2599122524261475, -1.0102970600128174, 1.6602520942687988, 0.5650651454925537, 0.667084813117981, -0.41639652848243713, -0.49429458379745483, -0.2841363847255707, 2.9117283821105957, 1.7624784708023071, -1.1906688213348389, 1.143791913986206, 0.5558678507804871, 0.4593953490257263, 0.04635265842080116, -1.352869987487793, -0.03719176724553108, -0.8885157108306885, -0.6894057393074036, 0.5219225883483887, 0.949339747428894, -1.7635347843170166, 1.1123743057250977, 0.9849374294281006, 0.7001460790634155, -0.43013542890548706, -0.2812533974647522, -0.16457828879356384, 0.7221754193305969, 1.6919195652008057, 0.16259318590164185, -0.26758116483688354, -0.7799367904663086, -0.7666398882865906, 2.1613497734069824, -0.9056225419044495, 1.5801881551742554, 0.7434824705123901, -0.8623101115226746, -0.9098297953605652, 0.9455482363700867, -0.9819967150688171, -1.326584815979004, -0.15298785269260406, 0.22916150093078613, 0.14009128510951996, -0.5308665037155151, -0.5144585967063904, -0.035067178308963776, 1.5000321865081787, -0.5059123039245605, -1.1401221752166748, -0.677228569984436, -0.8928433656692505, -1.0245038270950317, 0.376405268907547, -0.3821508288383484, 1.1618808507919312, -2.8875510692596436, -0.38338929414749146, -1.5458709001541138, 1.0491418838500977, -0.22578100860118866, 0.42335045337677, 0.5423194766044617, 0.349585622549057, -0.6610503792762756, -2.9231326580047607, -0.1094074696302414, -1.3510663509368896, -0.29030677676200867, 1.2409563064575195, -0.3745785057544708, 0.1472548395395279, 0.6952024102210999, 0.3602893352508545, 0.7931530475616455, 0.771775484085083, 0.3533305823802948, -0.24338020384311676, -1.4694911241531372, -0.2796405255794525, 0.7793165445327759, 0.27134498953819275, -1.496335506439209, -1.4631590843200684, -0.770806074142456, 0.8937803506851196, -0.17925311625003815, -0.43860483169555664, 0.9670879244804382, 1.1420931816101074, -0.449589341878891, 0.35549089312553406, -2.262566566467285, 1.520861029624939, 0.7235016226768494, 0.8466477990150452, -0.8723819255828857, 1.391310691833496, -0.7115845680236816, 0.06773117184638977, -0.5731713175773621, -0.4029950499534607, -1.0424044132232666, -0.8295859098434448, -2.020697593688965, 0.4935837388038635, 2.961071252822876, 0.032323990017175674, -1.1429362297058105, -1.089850902557373, -1.5537961721420288, -1.2030659914016724, 0.6726961135864258, -0.08096817880868912, -0.45452016592025757, 0.5927149057388306, -0.0008205744088627398, 0.48052361607551575, -0.8395867943763733, -0.04893582686781883, -0.30830979347229004, 0.08090442419052124, -0.4748895466327667, -0.4357070028781891, 1.4021527767181396, -0.7500178217887878, 0.19250953197479248, -0.4202779233455658, -0.4597738981246948, -0.082665354013443, 1.0177334547042847, -2.255314826965332], [0.5926743745803833, 0.4441654086112976, 0.049771588295698166, 0.9034146666526794, 0.34796780347824097, 0.807939887046814, 0.2590692937374115, -0.13306055963039398, -0.5167470574378967, 0.17678166925907135, -1.0028438568115234, -0.4004027843475342, -0.34761378169059753, 0.4686204195022583, 0.9299982786178589, 0.7680145502090454, -0.5086225271224976, 0.68929123878479, -1.012785792350769, 1.1350187063217163, -0.9272031784057617, 0.8140211701393127, -0.9159810543060303, -0.4356396496295929, -0.42754268646240234, 0.35034698247909546, 0.2670690417289734, 1.460316777229309, 0.7934831976890564, 0.47624731063842773, 0.7916754484176636, -1.9039044380187988, -0.614912211894989, 0.9173967242240906, -1.4650055170059204, 0.9260380268096924, -1.1052210330963135, -0.844753623008728, -0.8473011255264282, 1.292598009109497, 0.33944761753082275, 0.14655977487564087, 2.1632742881774902, -1.3642901182174683, 0.7033136487007141, 0.7034637331962585, 0.09631029516458511, -2.565899133682251, -1.1482281684875488, 0.6756731271743774, 0.039571184664964676, -0.23645226657390594, 2.534867763519287, 1.1604655981063843, -0.1992727369070053, -1.1332937479019165, 0.8295699954032898, -1.113281488418579, 0.9504832029342651, 1.4448341131210327, 0.5166876316070557, -0.8317825198173523, 1.2580251693725586, -0.855395495891571, -0.6556030511856079, -0.03538435325026512, 1.5751087665557861, 0.5106236934661865, -0.5729138255119324, 0.676001787185669, -0.8458741903305054, 0.47335198521614075, -0.25321507453918457, 1.0983833074569702, 0.13941295444965363, -1.7304133176803589, -0.8074182271957397, -0.7729992866516113, 1.8485589027404785, -0.8156396150588989, -0.3033502995967865, 1.9501737356185913, -0.4723307192325592, -1.6832656860351562, -0.3680817782878876, 1.323096513748169, 0.4598322808742523, 0.10669932514429092, -1.581586241722107, 0.26000526547431946, -1.212552547454834, -0.8558046221733093, 1.5221962928771973, 1.2392137050628662, -1.0771434307098389, -0.3851817548274994, 0.2664184272289276, -0.8722303509712219, -0.1956266164779663, -1.3644626140594482, -0.7690685987472534, 0.009813200682401657, 0.3407334089279175, 0.1016223356127739, -0.22368371486663818, -1.2610281705856323, -0.02194216288626194, -0.4560064375400543, -0.17313924431800842, 1.4068299531936646, 0.1120106652379036, -0.3425893783569336, 1.3867096900939941, 1.053945779800415, 3.124211072921753, 1.4091460704803467, -0.3091278076171875, -0.12654152512550354, -0.7587879300117493, -0.8653477430343628, -0.6307615041732788, -1.0627189874649048, -0.5293061137199402, -1.2173712253570557, -1.5961668491363525, -0.7065200209617615, 1.3654590845108032, -1.563987374305725, -0.16798587143421173, -1.2453535795211792, 1.1894333362579346, -1.9975489377975464, 0.9719409346580505, 0.014253812842071056, 0.6306765079498291, 0.25039294362068176, -0.5895235538482666, -0.39739906787872314, 0.0009844759479165077, 1.577035903930664, -1.3605239391326904, 1.6917394399642944, 0.045694880187511444, -0.49131104350090027, -0.28285858035087585, 1.0624713897705078, -2.1918935775756836, 1.2877779006958008, -1.4008370637893677, -0.8724762201309204, -0.4591813087463379, -0.413920521736145, 0.5990645885467529, -0.24442414939403534, -0.8633284568786621, -0.8207282423973083, -0.0024992539547383785, -0.5798407793045044, 2.281092405319214, -1.1952978372573853, 3.738239049911499, -0.11120102554559708, -1.2487889528274536, -0.7934247851371765, -1.6674017906188965, 1.9577971696853638, 1.2460498809814453, 0.9936227202415466, -0.420002281665802, -0.974701464176178, -1.2890762090682983, -0.8036808371543884, 0.13525007665157318, -1.7444216012954712, 1.687846064567566, -0.9516498446464539, 0.43778616189956665, -0.017759546637535095, -0.60499107837677, -1.3708441257476807, 0.024797003716230392, -0.7977520227432251, 0.09835362434387207, -0.5757275819778442, 0.25499293208122253, 2.002648115158081, 0.6202341318130493, -0.09881074726581573, 0.8264132142066956, -0.08163311332464218, 0.13587862253189087, 1.8604998588562012, 0.5304313898086548, 1.5226514339447021, 0.3849141597747803, 0.7322593927383423, 1.310219168663025, 0.8059070110321045, -0.20806171000003815, -0.09872598946094513, -0.6710929274559021, 0.11927235126495361, 1.357201337814331, 0.33041155338287354, 2.637331962585449, 0.08524386584758759, 0.22568728029727936, 0.013722475618124008, 0.625234842300415, 0.6185503602027893, 1.9032057523727417, 1.7087609767913818, 0.5337206125259399, -1.5743131637573242, 0.49423715472221375, 1.29788339138031, 0.6397111415863037, 0.9321771264076233, -0.4904051125049591, 2.3924427032470703, -0.5411065816879272, 1.054956078529358, -1.1428741216659546, -0.09917228668928146, 2.0976293087005615, -0.5258541107177734, -1.4822402000427246, 0.23868733644485474, -0.9444349408149719, 1.5656439065933228, 1.050861120223999, -0.8395532369613647, 0.2114715576171875, 0.48713526129722595, -0.3463376760482788, 0.32572004199028015, 1.6594501733779907, -0.07485345751047134, -1.0128343105316162, 1.5544381141662598, -1.3821648359298706, -0.49742820858955383, -2.7882297039031982, -0.0970122367143631, -0.41387903690338135, 1.262118935585022, -0.8428813219070435, -0.10876596719026566, 0.9315395951271057, -0.7123337984085083, 0.06191357225179672, 0.22668208181858063, -0.41741254925727844, -0.9556213021278381, 1.470264196395874, 1.5660642385482788, 0.6592535972595215, 0.16991284489631653, -0.3217330574989319, -0.09931117296218872, 0.3481782078742981, 0.8018885850906372, -0.3498303294181824, -1.1934670209884644, 0.6214257478713989, 1.683516263961792, 0.12233763188123703, 0.025194784626364708, -0.4841275215148926, 0.09567341208457947, -1.1958190202713013, -1.2413790225982666, -1.4487557411193848, 0.42247387766838074, 0.7186615467071533, -0.49377012252807617, 2.5868747234344482, 2.515814781188965, 0.4071298837661743, -0.34962478280067444, 0.03078608773648739, -1.8493645191192627, 1.6206296682357788, -0.6696570515632629, -0.33565714955329895, 1.06078040599823, 0.4678349196910858, -0.0034514109138399363, 0.06293104588985443, 2.7662765979766846, -0.16037513315677643, 1.1322135925292969, -0.11302382498979568, -0.12179698795080185, 1.7766077518463135, 1.0259557962417603, 0.8371408581733704, -0.031464219093322754, -0.404396116733551, 1.3338152170181274, 0.37028107047080994, 0.33680421113967896, -0.8437716960906982, 0.4991779923439026, 1.4138723611831665, -0.9396336078643799, -0.7312577962875366, -1.9096810817718506, 1.0819674730300903, 0.5767539739608765, -0.7731817960739136, -0.626301646232605, -0.15984006226062775, 0.7424615621566772, 1.001997709274292, -1.0116300582885742, -0.9259119629859924, -0.8093016147613525, 0.8409071564674377, -1.5518466234207153, -1.7688004970550537, 0.23120039701461792, -0.5519872307777405, -0.490410715341568, -0.23148928582668304, -0.4860767722129822, -2.823713779449463, -0.22779637575149536, -2.7956230640411377, 1.0372439622879028, 1.476833701133728, 0.5719231963157654, -0.04383260756731033, 0.8075540065765381, 0.6119884252548218, -0.24089622497558594, -1.0410429239273071, -0.6627371311187744, 1.0950711965560913, 0.5156075358390808, 0.5962474346160889, 0.11976157128810883, 1.7813694477081299, 0.17385458946228027, -0.9921877384185791, -0.6963552832603455, 0.9485741257667542, 0.28492051362991333, -0.5917670130729675, -1.0137178897857666, 0.9661858677864075, -0.8084530234336853, 1.3453401327133179, -0.6929671764373779, 1.2199811935424805, -0.019829925149679184, -0.5834226012229919, 1.0732295513153076, -0.24880248308181763, -0.12270194292068481, -0.54831463098526, 0.917479932308197, 0.5536447763442993, 0.39285245537757874, -0.4101616144180298, -0.13782256841659546, 0.45773211121559143, -0.44971951842308044, -0.33216774463653564, 1.212052583694458, -0.4636053442955017, 0.9978235363960266, -0.001274335547350347, 0.9812724590301514, 1.5100934505462646, 0.23640647530555725, -0.3454953134059906, -0.8461578488349915, 0.4554709196090698, 0.0048387073911726475, -0.13084058463573456, -0.6803575754165649, -0.6264550089836121, 0.0940399244427681, -1.7624422311782837, -0.022725559771060944, 0.7070373892784119, -0.32795342803001404, -0.748941957950592, -0.08391308039426804, -2.060800313949585, -0.38636764883995056, 1.3975366353988647, 0.7210285663604736, -1.6445121765136719, 0.3295069634914398, 0.34909167885780334, 0.809097170829773, -1.2683143615722656, -0.19161111116409302, -1.512868046760559, 0.680256724357605, 0.6955527663230896, -1.669974684715271, -1.6577643156051636, 0.8646126389503479, -0.28765228390693665, 0.22898057103157043, -0.042314257472753525, -1.1539989709854126, 0.786424994468689, -0.8890628814697266, 1.6959232091903687, 0.5459901094436646, -0.13829819858074188, 0.013790152966976166, 0.9641215205192566, -1.2049341201782227, 0.08400870859622955, -0.7443073987960815, -0.5682950615882874, 0.28413426876068115, 0.6054194569587708, 1.7283475399017334, 0.9773964881896973, -0.7629871368408203, -1.472861409187317, 0.41570982336997986, -0.31761664152145386, -0.01926473155617714, -0.23679079115390778, -1.3143224716186523, 0.11598168313503265, -1.3067846298217773, -0.5826440453529358, -0.4611820578575134, 0.5900300145149231, -1.1759895086288452, 1.4772281646728516, -0.7931926250457764, -0.7585570812225342, -0.13369932770729065, 1.8317502737045288, -0.39428800344467163, -0.12021095305681229, 0.2606657147407532, -0.5032230019569397, 0.24390418827533722, 0.9067505598068237, 0.07629712671041489, 0.2986183166503906, -2.0315639972686768, 0.03399666026234627, -0.38260573148727417, 0.4063851535320282, -0.16567502915859222, -0.8268328309059143, -1.4141088724136353, -1.0779573917388916, -1.4774112701416016, -0.8936569094657898, -0.4065244793891907, 0.3508264422416687, 0.3553384244441986, -0.18344956636428833, -0.899681568145752, 2.345069408416748, -0.6399003267288208, -1.8691812753677368, -0.40741491317749023, -0.0896025076508522, 0.21117347478866577, 0.8363598585128784, -0.6461253762245178, 0.806964099407196, 0.03560218960046768, 1.6982173919677734, -0.15403054654598236, 1.339664340019226, -2.0917470455169678, -1.1086820363998413, 1.4014012813568115, -1.6324001550674438, -1.4757475852966309, 0.14885994791984558, 0.054235901683568954, 1.1314945220947266, 0.18657052516937256, 0.44468045234680176, 0.09764626622200012, -1.4394865036010742, 0.5516061186790466, 1.762219786643982, -1.054198145866394, 0.4772767126560211, 1.916825771331787, -0.13117095828056335, -2.0315427780151367, -2.66898512840271, 0.4828932285308838, 0.011303399689495564, -0.6064678430557251, -1.9547375440597534, 0.8401890397071838, 0.5425457954406738, -0.17777302861213684, 0.4189012944698334, 0.05640649050474167, 1.9615939855575562, 0.15652911365032196, -0.6529532074928284, -2.2217586040496826, 1.0822710990905762, 1.933196783065796, -0.4628399908542633, -0.8970438838005066, -1.4797825813293457, 0.2557968497276306, 0.22248435020446777, -1.1860003471374512, 0.9805697798728943, 0.06549416482448578, 0.8417430520057678, 0.5189420580863953, -0.532057523727417, 0.5912314057350159, 0.6886735558509827, -0.5449569225311279, 0.9923409223556519, 0.804426372051239, 0.21262089908123016, -0.9139035940170288, -0.5768646001815796, 1.1658145189285278, 0.9149633049964905, 0.25368690490722656, -0.7977142333984375, -0.9486534595489502, 1.0177642107009888, -1.9551180601119995, 2.5338337421417236, 1.315639615058899, -1.8914389610290527, 0.07350733131170273, 0.47699058055877686, 2.1386680603027344, 0.13707031309604645, 2.246239423751831, 1.0895297527313232, -0.6475968956947327, 1.225401520729065, 0.5804305076599121, -1.1252632141113281, -0.27284738421440125, 0.8174138069152832, -0.1475130170583725, 0.6576830148696899, -0.6519389748573303, 0.09632276743650436, 1.0071449279785156, 0.5049377083778381, 1.8257795572280884, -1.381314992904663, -0.7968990802764893, -0.09317536652088165, -0.04102327302098274, 1.0263601541519165, -0.9354295134544373, -0.42759621143341064, 0.07907148450613022, 0.480214923620224, 0.2911289930343628, 0.6700345277786255, 1.0596277713775635, 0.1651640087366104, 0.10869530588388443, 0.5236777663230896, -0.03264571353793144, -2.0835952758789062, 1.9003766775131226, -1.36704421043396, 1.5562130212783813, -0.036340050399303436, 0.03859574347734451, -1.1227110624313354, -0.3339954912662506, -0.09030503779649734, 0.05726087465882301, -1.3115071058273315, 0.9265999794006348, 0.4301000237464905, 2.7261335849761963, -1.0836730003356934, -0.47202068567276, -0.5707175135612488, -0.3903568387031555, -0.9138725399971008, 0.28325992822647095, -1.3969314098358154, -0.24449455738067627, -1.3843364715576172, 0.7224956154823303, 1.0974940061569214, -1.9493974447250366, 0.6683516502380371, -0.0023662312887609005, 0.5626164078712463, 0.8108066320419312, 1.3289573192596436, -0.5763846635818481, 0.41333481669425964, 0.03589661419391632, -0.19830265641212463, -1.6809836626052856, -0.01845570281147957, 0.22759011387825012, 0.9400957822799683, 0.05294604226946831, 1.758080244064331, 0.5212311744689941, 0.4267581105232239, 0.7483827471733093, 1.1077593564987183, -0.5643284916877747, 0.29566630721092224, -1.3351446390151978, 0.1874629408121109, -0.015654508024454117, 0.44836318492889404, 1.2192448377609253, -1.2589952945709229, 0.7942176461219788, 1.4118324518203735, -0.040895551443099976, 0.7792375683784485, 0.4345070719718933, -1.5174518823623657, -0.08409927785396576, 0.20274201035499573, -0.05011620372533798, -0.8750120997428894, -0.04825460910797119, -1.065415382385254, -0.20536664128303528, -0.8893849849700928, 1.0023448467254639, -0.7048398852348328, -1.509588360786438, -1.0940040349960327, 0.36017122864723206, 0.7922947406768799, 2.955195426940918, 1.0762920379638672, 0.27565133571624756, 1.254332184791565, 0.5714971423149109, -0.1639261692762375, -2.398855447769165, -0.3243301510810852, 0.25028085708618164, -1.4453808069229126, 0.6019080877304077, -0.3083052933216095, -0.7874588370323181, 0.7380792498588562, -1.8190985918045044, -0.27087658643722534, -0.6618251204490662, -0.599521815776825, 1.3374114036560059, -0.5114046931266785, -0.30797719955444336, 0.6072762608528137, -0.6059731245040894, -1.0951405763626099, -0.7592930793762207, 1.424981951713562, -0.19370704889297485, 0.9559620022773743, 0.3412550687789917, 2.000840187072754, 1.5025718212127686, -1.7354198694229126, -0.8885930776596069, -1.0953550338745117, 1.2813258171081543, 0.7794199585914612, -0.792866051197052, 0.7225184440612793, -0.4149549901485443, 1.0754097700119019, -0.9744307994842529, 1.4407490491867065, 1.4745148420333862, 0.09025750309228897, 0.75572270154953, -0.3999381959438324, 0.9481505751609802, -1.027342438697815, 1.7498769760131836, -0.05611751973628998, 0.8597154021263123, 0.7742282748222351, -0.9754704833030701, -2.6880829334259033, 0.6440680623054504, 0.015550034120678902, -0.06623794138431549, -0.43049976229667664, -1.5860953330993652, 3.1557085514068604, 0.4894288182258606, -0.1894020438194275, 0.20384208858013153, 0.4197065234184265, 0.20654532313346863, -2.0454015731811523, 0.27510321140289307, -0.014310565777122974, 1.2123245000839233, -0.29849058389663696, -0.6536877751350403, 0.8487765192985535, 0.6559322476387024, -0.1948942095041275, 1.4635417461395264, -1.2268213033676147, -0.46200165152549744, -0.23226094245910645, 0.49674326181411743, 1.126261830329895, -0.9826911687850952, -0.2904987931251526, -0.5301406979560852, -0.12491694837808609, -0.25768622756004333, 0.04392482340335846, -0.5656441450119019, 0.05321674421429634, -0.36394718289375305, -0.5787187814712524, -1.1546857357025146, -0.576591968536377, 0.4594188630580902, -2.5303027629852295, 3.0653188228607178, 0.39766940474510193, -1.1741876602172852, 1.0762856006622314, -0.6541271805763245, 1.4406952857971191, 1.4201736450195312, 0.10095477104187012, -0.019955851137638092, 0.8875523805618286, 0.6878381967544556, 0.8696807026863098, -1.5219297409057617, 0.43944019079208374, 0.3046261668205261, 0.228713721036911, -0.8883028626441956, 0.699177622795105, 0.726922333240509, -0.4553781747817993, 0.007571217138320208, 0.9758744835853577, -1.6443068981170654, 0.6913353800773621, 0.34666869044303894, 1.1162364482879639, -0.7766475677490234, -0.42920175194740295, -1.036154866218567, -0.6223146319389343, -1.6195440292358398, 0.5971760153770447, -1.4431453943252563, 0.8728094696998596, -0.33674949407577515, -0.2515564262866974, -0.3954666554927826, -0.1518135368824005, 0.2327921986579895, 0.5480504631996155, -0.8884732723236084, 0.40221264958381653, -0.31543153524398804, -0.06710489839315414, -0.8518053889274597, -0.41994115710258484, -0.13578122854232788, -0.34452226758003235, 1.6156812906265259, 0.8902655243873596, -0.41232338547706604, 0.1443909853696823, 0.17814834415912628, -0.12591131031513214, -0.6816321015357971, -1.4108266830444336, -0.8468838334083557, 1.2714946269989014, -0.08074328303337097, 2.011658191680908, 0.36163994669914246, 0.17222794890403748, 1.465660572052002, 0.4164416790008545, -0.5452683568000793, 1.1831475496292114, -0.027493951842188835, -1.147308111190796, -0.7828161716461182, 0.34361732006073, 0.2797282040119171, -0.18380048871040344, 0.4390922784805298, -0.6250098943710327, 0.34008145332336426, -0.507296621799469, 0.26003333926200867, -0.3531948924064636, 0.079355388879776, -1.0444409847259521, -1.4286304712295532, 0.8833321928977966, -0.06421351432800293, 0.4689744710922241, 0.8275749683380127, -1.7537641525268555, 0.3176683783531189, -0.031947411596775055, -1.4318486452102661, 0.8302227258682251, -1.853480339050293, 1.4196537733078003, 0.035583123564720154, 1.3836089372634888, 0.8125618696212769, 0.22261464595794678, 0.37812215089797974, 0.0842026025056839, 0.8594685792922974, 0.18577852845191956, 0.4155508279800415, -0.6305119395256042, 0.16010192036628723, -0.6418680548667908, 1.0510071516036987, -0.9381725192070007, -1.5247368812561035, -0.06658182293176651, 0.16704922914505005, -0.011893197894096375, -0.37298384308815, 0.5877450704574585, -1.0524715185165405, -2.01094913482666, -0.017351774498820305, 2.278730869293213, -0.1532718688249588, -0.7030827403068542, -0.06897911429405212, 0.7131695747375488, 0.35224878787994385, 1.8050572872161865, 0.3471349775791168, 0.12568987905979156, -0.9132334589958191, -1.082399606704712, -0.6057168841362, 0.29099538922309875, -0.15469449758529663, -0.17541489005088806, 0.3411235213279724, -0.43180158734321594, -0.5976550579071045, 0.5558328628540039, 0.40964630246162415, -0.4508167803287506, -0.9899095296859741, 0.6339566111564636, -0.6822933554649353, -0.6840864419937134, -0.9922581315040588, -1.2848142385482788, 0.5226437449455261, 1.3602980375289917, 1.762695074081421, -0.5381220579147339, 1.8231614828109741, -0.7479808330535889, 0.14735092222690582, 1.323154330253601, 0.291604608297348, 1.5773160457611084, 0.5429778695106506, 0.12257619202136993, -0.8579472303390503, -0.8548315763473511, -0.372501015663147, -0.560444176197052, -0.6456375122070312, -2.480560064315796, 0.14676789939403534, 1.4071482419967651, -1.4385703802108765, 0.22302624583244324, 0.02387043461203575, -0.48499301075935364, 0.760892391204834, 1.328834891319275, -1.6517229080200195, 0.06936806440353394, 0.4460587501525879, -0.7065528035163879, 0.21748651564121246, 0.9983848333358765, -0.5608040690422058, 1.7908732891082764, -0.5440593361854553, 0.9683513045310974, -0.3846642076969147, 2.4219295978546143, 0.04754377901554108, -0.7353449463844299, -0.2736198306083679, 1.3913617134094238, 1.3142720460891724, -0.6685779690742493, -1.4757459163665771, -0.961145281791687, -1.5172680616378784, 1.0336004495620728, -0.6660204529762268, -1.1787139177322388, -0.5391805171966553, -0.2631012201309204, -1.222679853439331, 0.8522345423698425, -1.0703229904174805, 0.5250357985496521, 1.0575767755508423, -0.9445164203643799, -0.23498696088790894, 0.6260967254638672, -0.2755887806415558, -0.004154818598181009, -1.9789808988571167, -0.06795457750558853, 0.8714593052864075, 0.974865734577179, 1.9693185091018677, -0.32269367575645447, -0.41021430492401123, -1.1430637836456299, 1.3424782752990723, 1.3844120502471924, 0.34060123562812805, -0.014428464695811272, 1.0612866878509521, -0.85606849193573, 1.3881258964538574, -0.8312970399856567, -0.8079668283462524, -1.2077072858810425, -0.23161865770816803, -2.3148183822631836, 0.9165462255477905, -0.5772832632064819, -0.33418166637420654, -0.23886002600193024, -1.0670219659805298, -0.26345157623291016, -0.5236040353775024, 1.8793118000030518, -0.5915194153785706, -0.6090289950370789, 0.9080817699432373, -0.16134296357631683, -0.06763716042041779, 1.7511910200119019, -1.4263309240341187, 1.3981808423995972, 1.3203235864639282, -0.030332453548908234, 0.3738149106502533, -2.094146728515625, -0.19110336899757385, 0.8247679471969604, 0.7283607125282288, 0.4409681558609009, 0.8540679812431335, -0.5895002484321594, 1.0549883842468262, 2.5391528606414795, 0.28888753056526184, 0.3994465470314026, -0.11461860686540604, -0.9328866600990295, -0.13991154730319977, -2.0372729301452637, 0.5179319381713867, 0.03171661123633385, -0.11271751672029495, 0.16235876083374023, -0.6771983504295349, 0.18542541563510895, -0.10046947747468948, 0.266777366399765, -0.532924473285675, -0.28145331144332886, -2.5251851081848145, -0.024782584980130196, -0.7441268563270569, -0.19935055077075958, 0.055382173508405685, 0.9272135496139526, -1.7848029136657715, -1.541623592376709, -1.0968114137649536, -0.8050925135612488, 0.7427242994308472], [-0.4750141203403473, -0.2681346535682678, -0.12033020704984665, 1.0518553256988525, 0.22390547394752502, -1.3729506731033325, -0.3552616834640503, 0.3057608902454376, -1.2582237720489502, -0.8699742555618286, 0.15999281406402588, -0.616269588470459, -1.4850854873657227, -0.062039971351623535, -2.092054605484009, 1.234890103340149, 0.29125529527664185, -1.5388060808181763, 0.628574013710022, -0.7036993503570557, 0.6811262369155884, 0.13612687587738037, -0.7295278906822205, -0.831994354724884, 0.1552177518606186, -0.35898593068122864, 0.3634834885597229, 2.219008445739746, 0.4617067575454712, 0.1721159964799881, -0.5225227475166321, -0.19062770903110504, -0.9662585854530334, 0.24693676829338074, 0.43433845043182373, -1.1883697509765625, -1.2491713762283325, 0.8129255175590515, 0.3059522211551666, -0.6312307119369507, -0.23936216533184052, 0.8378418684005737, -1.5458126068115234, 1.8973658084869385, -2.2101173400878906, 0.26962485909461975, -1.308107614517212, -0.5428078770637512, -0.5712170600891113, -0.8889934420585632, 0.19660957157611847, 0.6946110129356384, -1.1635476350784302, 0.30573922395706177, 0.9631960988044739, 1.2838536500930786, 0.8764829635620117, 0.043052103370428085, -1.472814679145813, -0.02898663841187954, -0.031915754079818726, -0.00140752037987113, -1.3710839748382568, -2.731705904006958, -0.9674296975135803, -2.0505917072296143, -0.345142126083374, 1.9299862384796143, 0.3990114629268646, -0.9290032982826233, 1.0799981355667114, -0.5634200572967529, 0.1347968429327011, 0.20996157824993134, 0.21371214091777802, 0.5751519799232483, 1.5170542001724243, -1.2449871301651, 0.6021061539649963, -0.46295568346977234, 0.013103624805808067, 0.15515932440757751, 0.10459908843040466, 0.39134302735328674, -0.27664682269096375, 0.983752429485321, 0.07465863972902298, -0.6508805751800537, -1.3439662456512451, -0.5089997053146362, 1.1988706588745117, -0.9454399347305298, 0.6424046754837036, -0.21877960860729218, -0.2832397520542145, -0.20471641421318054, -0.15077830851078033, -0.6090111136436462, 0.42732781171798706, 0.4301234483718872, 1.0466804504394531, -0.4986875355243683, -0.19272446632385254, -0.9256396889686584, -0.5123103857040405, 1.9023288488388062, 0.034685470163822174, 1.734791874885559, -1.9355860948562622, 0.6196927428245544, 1.1960506439208984, -0.624435544013977, -1.0940296649932861, -0.18726933002471924, -0.35504698753356934, 0.33888810873031616, 0.9854912161827087, 0.44385647773742676, 0.8770514726638794, 1.8962599039077759, -0.8845333456993103, 0.006096979137510061, -0.6225217580795288, 1.0000911951065063, 0.5475331544876099, 0.9805925488471985, -0.9764290452003479, -0.02857132814824581, 1.276939868927002, 0.2274237871170044, -0.6616407632827759, -1.7439110279083252, 0.4273239076137543, -0.7340601682662964, -0.12674221396446228, 0.6824504733085632, 1.534867286682129, -0.06687688827514648, 0.6755756139755249, 1.1478389501571655, -1.0017266273498535, 0.37356922030448914, 0.38104087114334106, 0.2003813534975052, 1.123397707939148, -0.9153800010681152, 0.4797881841659546, -0.1633305549621582, 0.22630003094673157, -0.9872162342071533, 1.0345690250396729, 0.5004266500473022, -0.699198305606842, 0.032727908343076706, -0.7419117093086243, -1.4184165000915527, -0.26848936080932617, 0.7388439178466797, 0.054309722036123276, 1.2976770401000977, 1.1702290773391724, -1.5411525964736938, 1.7180150747299194, 0.8842684030532837, -0.4298931956291199, -0.043684132397174835, -0.49045678973197937, -0.5930309891700745, 1.047309398651123, 0.34804752469062805, 1.2244517803192139, 0.8448548913002014, -0.5353034734725952, -0.9662289619445801, 0.6240416765213013, -2.547734498977661, 1.686253547668457, -1.5613678693771362, -1.0404901504516602, -0.6129052042961121, -0.516255795955658, 2.1597588062286377, -0.39020615816116333, 1.4506630897521973, -1.887625813484192, -0.1759916990995407, 1.3830264806747437, 1.1320632696151733, 1.2789504528045654, 0.25489723682403564, -1.0998897552490234, -0.5213608145713806, 1.5432157516479492, 0.675611138343811, 2.235433578491211, -1.2459508180618286, -1.2518961429595947, 1.9915217161178589, 1.7185468673706055, 2.3819355964660645, -0.4530649781227112, 1.1572380065917969, 0.25727057456970215, 0.1354304403066635, -0.4636811912059784, -0.4414534568786621, 0.9500864148139954, 0.7855234146118164, 0.025334246456623077, -0.12147312611341476, -1.9984074831008911, -0.6721332669258118, 1.8177192211151123, 1.4880516529083252, -0.8914278745651245, -0.59714674949646, -0.5259782075881958, -0.3539784550666809, 0.7700439691543579, 0.6232338547706604, -1.2705832719802856, -0.1310095489025116, 0.3770677149295807, -1.5249285697937012, -0.19163571298122406, 2.741957664489746, 0.4170437157154083, 0.2030714899301529, 1.4298149347305298, -0.507472574710846, -0.15616557002067566, -0.38554149866104126, -0.6728227138519287, 0.7236356735229492, -0.6896734833717346, 1.4986200332641602, -0.7085762023925781, 0.1250133216381073, 0.027393421158194542, 1.555380940437317, -1.1787575483322144, 0.2595260441303253, -2.9987244606018066, -0.6456779837608337, -1.2068829536437988, 0.9734807014465332, 1.836488962173462, 0.5681450963020325, -1.0213508605957031, 0.7065033316612244, 0.5633488297462463, -0.36664915084838867, 2.554165840148926, 0.3212248980998993, 0.6996383666992188, 1.153512954711914, -1.0771256685256958, -1.362532377243042, 1.303367257118225, -1.7184171676635742, -0.6097322702407837, -0.4856402277946472, 0.18582874536514282, -0.14405615627765656, -1.1322797536849976, 1.060697078704834, 0.8710965514183044, 0.5782859921455383, 2.169443130493164, -0.6324330568313599, 1.706302285194397, 0.04983166232705116, 0.2015371024608612, -0.5063219666481018, 0.8847399353981018, -0.7130950093269348, -1.9612481594085693, 0.24901644885540009, -0.23209448158740997, 1.0380537509918213, 0.5010862946510315, -0.7804012894630432, 0.2569841146469116, -0.8616665601730347, 0.0720532089471817, -0.07386520504951477, -0.22840562462806702, -0.6613535284996033, 0.37871280312538147, 1.0841612815856934, 0.5071345567703247, -0.4470430314540863, -1.0347399711608887, 0.4120366871356964, -0.1766655445098877, 0.056674711406230927, 0.10115563869476318, 1.6700716018676758, -1.0454308986663818, -2.230229377746582, -1.264613389968872, -0.7767072319984436, 0.02325478009879589, -1.882120966911316, -0.9612822532653809, 0.6085720658302307, -0.19722560048103333, 0.8290591239929199, -0.5418005585670471, 0.6996604800224304, 0.5146095156669617, 0.38978826999664307, 0.5552433133125305, 0.16876383125782013, 1.3066108226776123, 0.8405557870864868, 1.0946482419967651, -1.3226927518844604, -1.0554594993591309, -0.6970258355140686, 1.0103098154067993, -1.1610512733459473, 0.41791868209838867, 1.3078255653381348, 0.27350524067878723, 0.10734307765960693, 0.10974734276533127, -1.4833407402038574, 1.5128061771392822, 0.19274328649044037, 0.650510847568512, -0.2433616667985916, -2.208521604537964, -1.5474032163619995, 0.8703709244728088, -1.4446464776992798, 0.3463892340660095, 0.49709421396255493, -0.6278827786445618, -0.3766787052154541, 1.8833723068237305, -0.17346051335334778, -0.05712046101689339, 0.6781993508338928, 1.2410482168197632, -0.06665501743555069, 0.6378034353256226, -0.6581695079803467, 1.7258234024047852, -0.7701792120933533, 0.6399290561676025, -0.03775835409760475, 1.0602755546569824, -1.9521629810333252, -1.1740399599075317, 0.9318791031837463, 1.5284669399261475, -1.2172067165374756, 1.6261699199676514, -0.6340161561965942, 0.8968066573143005, -1.1275370121002197, 0.9125370383262634, 0.10512619465589523, 0.07739180326461792, 0.32785332202911377, 0.6111481785774231, 1.0115289688110352, 0.034197818487882614, 1.4159194231033325, -1.8023197650909424, 0.7882930040359497, 1.5434114933013916, -1.7904727458953857, 1.4279967546463013, -0.3038225471973419, 1.121665120124817, 2.3496477603912354, 1.2401102781295776, 1.387381672859192, 0.2508513927459717, -1.4884731769561768, 0.23778636753559113, 1.102757453918457, -1.6203778982162476, -1.1773457527160645, 0.01806638576090336, 0.18469980359077454, -0.7785840034484863, 0.6857544779777527, -1.0047351121902466, 0.38450902700424194, 1.7715418338775635, 1.4388571977615356, 1.6796469688415527, -1.5671168565750122, -0.5015856623649597, -0.31643182039260864, 0.013291149400174618, 0.697594404220581, 0.7769092321395874, -1.0514570474624634, 0.2087225615978241, 0.32954615354537964, -0.620846688747406, -1.082698941230774, -1.5630015134811401, 1.765305995941162, -0.12656159698963165, 0.7236807942390442, -1.1732885837554932, 0.4628165662288666, 0.5548797845840454, -0.8588835597038269, 0.47156625986099243, 0.2670435607433319, -0.4577258229255676, -1.7266428470611572, 0.8266633152961731, -0.6372939348220825, 2.208357334136963, 1.1469489336013794, -0.6878743767738342, 0.6987618803977966, 0.9205052256584167, 0.41280102729797363, -1.2169106006622314, 0.6943403482437134, -1.7510319948196411, 0.39148518443107605, 0.2013225555419922, 1.5280643701553345, -0.37481993436813354, 0.8005985021591187, 1.5202504396438599, -1.942670226097107, 1.8001142740249634, 0.26455777883529663, 0.02685227245092392, -0.9169466495513916, 0.2541056275367737, 2.2620179653167725, 0.6576240062713623, -2.827319622039795, -0.3862105906009674, 0.37695175409317017, -0.6839275360107422, 0.7539434432983398, -0.6811567544937134, 0.7161645293235779, 1.289901614189148, 0.599674642086029, -0.8441473841667175, -0.42016083002090454, -1.024506688117981, 0.7921113967895508, -1.2610771656036377, -1.6082807779312134, -0.042897891253232956, -0.9378973841667175, 0.4157950282096863, -0.10003858804702759, -0.3889560401439667, -1.3705521821975708, 0.5197414755821228, -0.15995293855667114, -1.4784696102142334, 1.915823221206665, -0.6240407228469849, 0.3796153664588928, 0.5644484162330627, -0.8418970704078674, -1.2458807229995728, -0.39956676959991455, 0.763724148273468, -1.017526626586914, 0.8152717351913452, 0.21603986620903015, 1.4549766778945923, -1.192556381225586, 0.6309179663658142, -1.466444969177246, 0.6794983744621277, 0.20929603278636932, -0.5584598779678345, -0.2235894501209259, -0.5417557954788208, -0.12270765751600266, 0.05801078677177429, 1.3619797229766846, -0.5119702219963074, 1.1872742176055908, 0.5812652111053467, -0.2949734330177307, -0.6198898553848267, -0.041877977550029755, 1.1599160432815552, 2.233222246170044, 0.7804089784622192, -0.3337792158126831, -0.19756647944450378, -1.3279016017913818, -0.33520370721817017, 1.6872775554656982, 0.2416948825120926, -0.013685372658073902, 0.673658013343811, -0.17802536487579346, 0.5925130248069763, -1.8376095294952393, 1.3581938743591309, 1.4548397064208984, -1.0222008228302002, -0.6516546010971069, 1.0634125471115112, -0.02498273178935051, 0.6460769772529602, -1.6555237770080566, 0.4737837016582489, -0.8540928363800049, 0.5864753127098083, -0.5828041434288025, -1.0626780986785889, 0.3296944200992584, 0.0921248346567154, -0.26891082525253296, 0.7706753015518188, 0.4800339937210083, -1.571418285369873, 0.18475522100925446, 0.428404301404953, -0.9715153574943542, -1.099593162536621, 0.12736088037490845, 1.088690161705017, 1.236602544784546, -1.2227696180343628, -0.23838616907596588, 0.45688366889953613, -0.14830146729946136, 0.00083872553659603, 0.048518162220716476, -0.06992410868406296, -1.44528067111969, -0.7484888434410095, -0.30679890513420105, 0.18234355747699738, 0.5032601356506348, -2.153535842895508, 0.6214919686317444, -0.5091423988342285, 0.5364379286766052, 0.9160651564598083, -0.030859148129820824, 0.44634994864463806, -1.28441321849823, -0.6406400203704834, 0.1584576964378357, -0.3612172603607178, -1.2142393589019775, 0.46092289686203003, 0.29574134945869446, 0.5589085221290588, 0.40004608035087585, 1.6123408079147339, 0.8158822059631348, 1.2170984745025635, 1.0759954452514648, -1.4640194177627563, -1.2548316717147827, -0.6556428670883179, -0.5742665529251099, -0.09443310648202896, -1.3893615007400513, -0.9628732800483704, 1.3716384172439575, -0.10868088901042938, 0.770020067691803, 0.36115726828575134, -0.478643000125885, -1.0509833097457886, -0.870884895324707, 0.8415308594703674, 1.42141592502594, -0.46932342648506165, 2.0238778591156006, 0.33682671189308167, -0.7368320226669312, 0.7662558555603027, 0.20043590664863586, 1.697299838066101, -0.2861797511577606, -0.07383851706981659, -1.1688708066940308, -1.0943373441696167, 0.36890238523483276, -0.7380920052528381, 1.5080517530441284, -0.7504380345344543, 1.2775042057037354, 0.11756480485200882, 2.720632553100586, -0.8506070375442505, -0.49105846881866455, 0.15454073250293732, -0.5675543546676636, -0.8621365427970886, -1.697546362876892, 0.6424757242202759, 1.2081012725830078, 0.9296402335166931, 0.17890578508377075, -0.1623629480600357, -0.22372333705425262, 1.2257105112075806, -0.3644387125968933, 0.7837449908256531, -0.6377508044242859, -0.3397676944732666, -0.3716825842857361, -0.9272550940513611, 1.4955716133117676, -0.7918035984039307, 0.5300539135932922, -1.542542815208435, 2.1553969383239746, 0.17677071690559387, 0.4315735697746277, -0.2462536096572876, -1.3399477005004883, 0.5977532267570496, 1.5224415063858032, 0.8215497732162476, 0.49723485112190247, -0.3951973617076874, 0.2697295546531677, -1.3649777173995972, -0.03254149481654167, 1.0646007061004639, 1.2171335220336914, 1.2455164194107056, -0.02757451869547367, -1.3747931718826294, -0.12672655284404755, -1.6834477186203003, -0.6749383211135864, 1.4608614444732666, -0.5907000303268433, -0.9568523168563843, -0.3559674918651581, -0.07060643285512924, -0.9222938418388367, 1.7507466077804565, -2.3101394176483154, 0.5449346303939819, 1.709047555923462, -0.3129395544528961, 0.6449689865112305, 0.4723978638648987, 0.5785375833511353, 1.886919617652893, -0.03820451721549034, -0.3815784752368927, -1.6097750663757324, -0.021188294515013695, -0.7300164699554443, -0.20555947721004486, 0.4576813876628876, 0.9958994388580322, -2.3323419094085693, 1.8419382572174072, -0.10577020794153214, -0.5564218163490295, 1.6851211786270142, -0.8870848417282104, -2.052248001098633, 0.17024125158786774, 0.5389904379844666, 0.8481407165527344, -1.8083029985427856, 2.1497695446014404, 0.026534968987107277, -2.0574235916137695, -0.7138346433639526, 1.0774565935134888, -0.18114303052425385, 1.6390949487686157, -0.08733908087015152, -0.3433043360710144, -0.9646716713905334, -0.5009123682975769, -1.7736564874649048, 1.2092974185943604, 0.8228268027305603, 1.223686695098877, -0.7540386319160461, -1.744340419769287, -0.22458863258361816, -0.3352479636669159, -0.7313966751098633, -1.3479244709014893, -0.6083886623382568, 0.9955999851226807, -0.11379913240671158, 0.20569521188735962, -2.1022183895111084, 0.621273934841156, 1.565030813217163, 0.9093566536903381, 0.03721539303660393, -0.5063009262084961, 1.1443349123001099, 0.1265643984079361, -0.7898033857345581, 0.00831935927271843, -1.2763913869857788, 0.7444835305213928, 0.6462350487709045, 0.6104518175125122, 1.3200074434280396, -0.6500393152236938, 0.24936485290527344, -0.1339680701494217, 2.265489101409912, 0.44223183393478394, -0.07266448438167572, -0.37184229493141174, -0.2771708071231842, -0.16478683054447174, 2.091663122177124, -1.2726447582244873, -1.5490052700042725, 0.1157737746834755, -1.8831897974014282, -0.9671007394790649, -0.2699408233165741, -0.4543929398059845, -1.6695098876953125, 0.4788190722465515, 1.4266366958618164, -0.8181726932525635, 0.5552383065223694, 1.5008658170700073, 0.7523341178894043, 1.3365895748138428, 0.6687804460525513, 0.8953824043273926, -0.38424545526504517, -0.21809880435466766, 0.18174844980239868, 1.386778712272644, 0.26293739676475525, -0.5857406854629517, 0.3719235956668854, -1.175583004951477, -0.5107100009918213, 1.8474520444869995, -0.2158796489238739, 2.1511433124542236, -1.5524882078170776, -0.1410035640001297, -1.395853877067566, -0.0610007718205452, -0.027655674144625664, 0.4328318238258362, -0.0019015673315152526, -0.09255679696798325, 0.5320255160331726, -1.016948938369751, -1.4327260255813599, -0.6655361652374268, -0.3887755572795868, 0.09233902394771576, 0.822160005569458, 1.7449630498886108, -1.1309113502502441, -0.05648112669587135, -1.951926827430725, 0.5535949468612671, 0.15432488918304443, 0.035387132316827774, 1.9263575077056885, -1.0527324676513672, -0.0075874305330216885, 1.0665706396102905, 0.10654843598604202, 0.10816720128059387, -0.7198939323425293, -1.3676854372024536, 0.2180376499891281, 0.48789018392562866, 1.9723293781280518, 0.40196627378463745, 0.6292667388916016, 1.0201612710952759, -2.6061389446258545, 0.2548528015613556, 0.2916199266910553, 1.6728781461715698, 0.45602181553840637, 1.2011749744415283, 0.09882069379091263, 0.22046712040901184, -1.919449806213379, 0.9545050263404846, -1.2553248405456543, -2.0488719940185547, 1.026729941368103, 1.1592819690704346, 0.4877954423427582, 0.6159061193466187, 0.40002894401550293, -1.3832179307937622, -2.1046464443206787, -1.089345932006836, -1.0602697134017944, -0.5045633316040039, -0.5842897295951843, -1.3553669452667236, -1.0766125917434692, -0.6794994473457336, 0.4503781795501709, 0.9630683660507202, 0.09426852315664291, -1.0613402128219604, 0.0564383938908577, -0.05479273572564125, 0.16785258054733276, 0.4301094114780426, 1.3893738985061646, -0.0917535200715065, 0.6393011808395386, -0.3754684329032898, -1.3004662990570068, 2.9651083946228027, -1.314299464225769, 0.10342174023389816, -0.8517686724662781, -1.8785884380340576, 0.4332810938358307, 0.4352436661720276, 1.499686360359192, 0.38476765155792236, -0.40630239248275757, -1.347429871559143, -1.7629845142364502, 0.5371941328048706, -0.716212809085846, -1.6207025051116943, 0.35564377903938293, 0.8601938486099243, -0.3489069938659668, 0.5322899222373962, -0.7767764329910278, 0.8525204658508301, 0.18585851788520813, -0.663318395614624, 0.5932000875473022, 0.26866212487220764, -0.01808573678135872, -0.514485239982605, -0.7169942259788513, -0.03171537443995476, 1.49474036693573, -1.6137641668319702, -0.016889410093426704, -0.19736957550048828, -0.246709406375885, 0.7148513197898865, -0.15678371489048004, 0.7706142663955688, -0.13324935734272003, -2.311282157897949, 0.5651249885559082, 0.946609616279602, 0.1610139012336731, -0.1791696697473526, 0.7901678085327148, 2.2190709114074707, 1.3097243309020996, -0.6319376230239868, -0.4327446222305298, 0.6833457946777344, 1.5336309671401978, -1.2603418827056885, -1.2310720682144165, 1.390560507774353, -0.7706472277641296, -1.140732765197754, -0.4106254577636719, 0.058227572590112686, 0.3031368553638458, -0.664053738117218, -0.48145249485969543, -1.312482237815857, -0.014858977869153023, -0.37920477986335754, -0.31251344084739685, -1.4186969995498657, -0.4541584849357605, 0.26462969183921814, -0.6356650590896606, 0.9961366057395935, 0.20122848451137543, -0.9922983050346375, -2.3963088989257812, -0.5291250348091125, 1.4378130435943604, -1.0189741849899292, 0.2947699725627899, -1.0200247764587402, -1.9467213153839111, 0.657120406627655, -1.3806581497192383, 0.2450663298368454, 1.7914458513259888, 0.7277616858482361, -0.01945173740386963, 1.5793633460998535, -1.5408905744552612, -0.826274573802948, -0.46727728843688965, -0.08560436218976974, -0.18344098329544067, -1.7162694931030273, -0.001196029013954103, 1.4251601696014404, -0.29904547333717346, 0.29769080877304077, 0.4214927852153778, 0.027968263253569603, 0.012811167165637016, 0.4787982702255249, -0.47760888934135437, 0.5877993106842041, 1.0993269681930542, -0.2485382854938507, 0.8109338879585266, -0.302461177110672, 0.53377366065979, -0.9714773297309875, 1.2473018169403076, -0.13623516261577606, 0.7016443610191345, 0.3319736421108246, 0.18972375988960266, 0.9385995864868164, -0.3991558253765106, -0.9677843451499939, 0.1116752028465271, 1.5736156702041626, 0.22382918000221252, 1.1367361545562744, -0.22008153796195984, 0.02839749865233898, 0.7029207348823547, 0.042905498296022415, 0.5224902629852295, -1.2052642107009888, -0.35201019048690796, -0.19946368038654327, 1.6556615829467773, -0.568498432636261, -1.2546156644821167, -0.5992512106895447, -1.0271821022033691, 0.6397366523742676, 2.1815834045410156, -1.7571018934249878, -0.18635433912277222, 1.458081841468811, -0.5082151293754578, 0.5083202123641968, -0.21026988327503204, 1.0142521858215332, -0.045574482530355453, -2.288490056991577, -0.12081653624773026, 0.10870020091533661, -0.6244681477546692, -0.5635303854942322, 1.7821522951126099, -0.9984741806983948, 1.0263844728469849, -0.5943000912666321, 0.15875302255153656, 0.8210886120796204, 0.4437241852283478, -0.9038475751876831, 1.2124345302581787, -0.7950133085250854, -0.17404265701770782, -2.8685407638549805, -1.1238080263137817, -1.1444859504699707, 0.44621542096138, -1.2228050231933594, 0.1488412618637085, 2.664450168609619, 1.581555962562561, 1.2039456367492676, 0.4326809048652649, 0.6207192540168762, -0.4619055390357971, -0.11856937408447266, -2.228668451309204, 0.14377237856388092, 0.5899834632873535, 1.9496537446975708, -0.6071958541870117, 1.893420934677124, -0.31037524342536926, 0.33288252353668213, -0.02776133082807064, 0.16721414029598236, -0.33350658416748047, 0.4769880771636963, -0.9756121635437012, 0.8540621399879456, 0.813960611820221, 1.0753389596939087, -0.01906491629779339, -0.25382813811302185, 0.9374737739562988], [0.3152415156364441, -0.5032077431678772, 0.6780336499214172, 2.2421746253967285, -1.2567845582962036, -1.6723339557647705, 1.7628099918365479, 1.2463703155517578, -0.42799314856529236, 0.3856410086154938, 0.30615323781967163, 1.8073840141296387, 0.23224565386772156, 0.7494070529937744, -0.8935779929161072, -0.8580033183097839, 0.04827914759516716, 0.290127694606781, -0.6950892210006714, 1.0050162076950073, -1.2845410108566284, 1.1011775732040405, -1.0497409105300903, 0.5715560913085938, 0.12079325318336487, 2.2053799629211426, -1.1320502758026123, -2.1497111320495605, -0.7377498149871826, 0.34328004717826843, 2.321934461593628, 1.3150168657302856, -0.2196679413318634, -0.7711052298545837, -0.7600652575492859, -1.7728804349899292, -0.21899722516536713, 0.22151727974414825, -1.657753586769104, 0.13664542138576508, -0.11143064498901367, 1.121263027191162, 0.8580280542373657, 0.8125524520874023, -2.744642734527588, 0.8806350231170654, -1.779387354850769, -2.385342597961426, 0.03753950446844101, -1.327905535697937, 0.8065038919448853, -1.2253843545913696, 0.09278164058923721, -0.15005408227443695, 0.8424503803253174, -2.7436916828155518, 0.49040547013282776, 0.5602426528930664, -1.442686676979065, 1.0314010381698608, 0.3706956207752228, 2.101646661758423, 0.4692918360233307, 0.1432380974292755, -1.8098644018173218, -0.2231537252664566, -0.716593861579895, -1.1216163635253906, 1.131296992301941, -0.6024876236915588, 0.03706122934818268, 0.7002196907997131, 0.39709600806236267, -0.2520930767059326, 0.47581997513771057, 1.1214218139648438, 0.019932663068175316, 0.6752005219459534, -0.3974543511867523, 0.5901303887367249, 0.5352553725242615, -0.8450156450271606, 0.3356965184211731, -0.45012810826301575, -1.993913173675537, 0.45977407693862915, 1.039743423461914, 1.1809369325637817, 0.9329649806022644, 0.14610446989536285, -1.5186407566070557, -0.5253323912620544, 0.8388076424598694, 0.29398101568222046, 0.8436084985733032, 0.3040938079357147, 0.21490687131881714, -0.8713592290878296, -0.17630048096179962, 0.37333038449287415, 0.12529638409614563, 1.2191166877746582, 0.4860127866268158, 1.07485032081604, -1.1213326454162598, 0.6346635818481445, -0.5976720452308655, -0.42955827713012695, -1.2157012224197388, 0.864810049533844, 0.7518113851547241, -0.6317490339279175, 1.243183970451355, -1.1169532537460327, -0.344052255153656, -0.029579438269138336, -1.014708161354065, -0.6995666027069092, 1.21821928024292, -0.09816992282867432, -1.6914478540420532, 0.4062199592590332, 0.8550736308097839, -0.48074477910995483, 0.5015390515327454, 1.498368501663208, -0.8011602163314819, 0.7228339314460754, -0.5869641304016113, 1.0983858108520508, -0.8011832237243652, -0.28862547874450684, -0.28714719414711, 0.8604568243026733, -0.2522176504135132, -0.6078900694847107, 1.1314935684204102, -0.7021722197532654, -0.21098901331424713, 0.989819347858429, 1.385180950164795, -1.5549942255020142, 0.8346011638641357, 0.4332639276981354, -0.2639675438404083, -0.8734080195426941, 0.17489609122276306, 0.1505582332611084, -0.5245991945266724, 1.1131280660629272, 0.6643577814102173, -0.9451310634613037, -0.3285261392593384, 1.0871002674102783, 1.8702048063278198, -0.12358202040195465, -0.05882754921913147, 0.4782178997993469, -1.2938450574874878, 0.96321702003479, 0.9190837144851685, -0.49721527099609375, 0.21398036181926727, -0.3821680545806885, 0.6839416027069092, -0.6372638940811157, 1.0654476881027222, 0.7996763586997986, 0.3381262719631195, -1.2231022119522095, -1.3225172758102417, 0.06271550804376602, 0.35380688309669495, -0.9257817268371582, 1.9272713661193848, -1.839341163635254, -0.06475788354873657, -2.3488385677337646, 0.4843757152557373, 0.04053613916039467, -1.6672277450561523, -0.3411693871021271, -1.1888093948364258, -0.9523151516914368, 0.5838841795921326, -1.9304908514022827, 0.23675456643104553, 0.6916810870170593, -1.5849412679672241, 0.10712460428476334, 0.9833747744560242, 0.7164043188095093, -0.04454273357987404, -0.5533398389816284, 1.0804775953292847, -0.7245323061943054, -1.0302402973175049, 1.4592329263687134, 0.0589812733232975, -0.9647974371910095, 2.1881492137908936, -1.086236834526062, 0.2605861723423004, -0.41626277565956116, 1.7298082113265991, -0.12341701239347458, -1.713799238204956, 0.2519720792770386, 0.5368065237998962, -0.5031334161758423, 2.2375032901763916, -0.29723691940307617, 0.6828635334968567, 0.8506115078926086, -0.19679465889930725, -0.6720489859580994, -1.9378021955490112, -0.5048391222953796, -1.3056175708770752, 0.03316837176680565, 0.018301470205187798, -0.38187581300735474, -1.3425906896591187, 0.5949130058288574, -0.09844201058149338, -1.4835844039916992, 0.3897687792778015, 0.1206049919128418, 0.8562883734703064, 1.4533635377883911, -0.0657481849193573, 0.5135263204574585, -0.41599634289741516, 0.2974717319011688, 0.2130943238735199, 1.1128854751586914, -0.11384136974811554, 0.13038071990013123, 0.6680080890655518, 0.5861624479293823, -2.3454580307006836, 1.6001056432724, -1.961405873298645, 0.6517401337623596, 1.4045159816741943, -0.9863021373748779, 0.47085022926330566, -0.9432812929153442, -0.34074974060058594, -0.8310419321060181, -1.8606903553009033, 1.0325065851211548, -0.3725947439670563, -0.8478002548217773, 1.1340970993041992, 0.0828319787979126, 1.4819746017456055, -0.9235570430755615, 1.1883416175842285, -0.25016337633132935, -0.886124849319458, 0.17887890338897705, 0.5633955597877502, -1.0495673418045044, -0.20132997632026672, -0.8859083652496338, -0.1486889123916626, 0.4794745147228241, 1.1820834875106812, 0.08374175429344177, 0.9328667521476746, 0.9994146823883057, -1.3575633764266968, -1.4279648065567017, 0.8416889905929565, 1.4022542238235474, -0.37237465381622314, 0.585938036441803, -0.8158354759216309, -0.45932695269584656, 0.14737124741077423, 2.5527985095977783, -0.21842212975025177, 1.5568610429763794, 0.5490559935569763, 0.9618119597434998, 0.2920222580432892, 0.8345707654953003, 1.6974279880523682, -0.34707191586494446, 1.1259567737579346, -0.030250290408730507, -0.4065578877925873, -1.5153251886367798, 1.2967264652252197, -0.16767475008964539, 0.32215380668640137, 2.9568092823028564, -0.27082327008247375, 0.6417698264122009, -0.44368690252304077, -0.5878114700317383, -0.5072145462036133, 0.15994712710380554, 0.8120214939117432, -1.1194053888320923, -0.6266553997993469, 0.4333570599555969, -0.7158144116401672, -1.8703049421310425, 0.9872887134552002, 0.5936039686203003, 0.02542768232524395, -0.3624100685119629, -0.3039151132106781, 0.30112984776496887, -0.4144672453403473, -0.2788691818714142, -0.5157625675201416, -0.20317302644252777, -1.5136172771453857, -0.6463797688484192, 0.497900128364563, -0.5002321004867554, -2.4298653602600098, 0.5775066614151001, 0.24275079369544983, -1.0251671075820923, -0.744103729724884, 0.29866117238998413, -0.45691820979118347, -0.15359783172607422, 1.602046012878418, 0.2540564239025116, -0.6018573045730591, -0.6676085591316223, 1.4703419208526611, -0.07191438227891922, -1.9828028678894043, 2.2044575214385986, -1.312587022781372, -0.9837648868560791, 0.4325721263885498, 0.08607883006334305, 0.8545615077018738, -0.6794198155403137, 0.011115333996713161, -0.4974934458732605, -0.3998139798641205, -0.7797169089317322, 1.4609720706939697, -2.0200424194335938, -0.1905066817998886, 0.38994720578193665, -0.6271753907203674, -0.724231481552124, -0.3146679103374481, -0.691765546798706, -1.7315047979354858, -0.09807763993740082, -0.24310709536075592, -0.606753408908844, 0.3093198537826538, -0.06093812361359596, 0.23196780681610107, -1.1734410524368286, 0.26356446743011475, 0.21728748083114624, 1.3493382930755615, 2.247185230255127, 1.238282322883606, 2.1649510860443115, -1.0474772453308105, 0.8018977046012878, -0.7388994097709656, 0.009725771844387054, 0.6965377330780029, 0.5089210867881775, 1.1843242645263672, 0.6291137337684631, -0.1162128895521164, -0.23816461861133575, -1.781987190246582, 0.28625205159187317, 0.6483510136604309, -1.1301569938659668, -0.145822674036026, 0.5904253125190735, -0.5274550914764404, -0.328556090593338, 1.1963436603546143, -0.2676493227481842, -0.6061310768127441, 0.26792773604393005, -1.6762861013412476, 0.2038036733865738, -0.4675491154193878, -0.47849565744400024, -1.9779585599899292, -0.8501660823822021, -1.639461874961853, -1.7129929065704346, -0.12343782186508179, 1.733242392539978, -0.34982192516326904, -0.6975044012069702, 0.10966740548610687, -0.011596924625337124, 0.9009978771209717, -0.25007033348083496, 0.08545957505702972, 1.603990912437439, 1.0200680494308472, 0.23933042585849762, -0.11449804902076721, 0.4598921239376068, -0.2875267565250397, 0.10148496925830841, -0.06782343238592148, -0.5064329504966736, 1.2816356420516968, 0.19675491750240326, -1.537584900856018, 0.42420756816864014, -0.8949008584022522, -0.5506345629692078, -0.6080072522163391, 2.6680121421813965, 1.5460035800933838, -0.7231954336166382, -0.11507729440927505, -1.3434312343597412, 1.4283990859985352, -0.9709596633911133, 0.2366752326488495, 1.0007480382919312, 0.21477799117565155, -2.5220165252685547, -0.7542669177055359, 0.1858150213956833, 0.8894603252410889, 0.3629787862300873, 0.23314614593982697, 1.2563056945800781, -2.196699857711792, -0.5347470641136169, 1.2639739513397217, -0.9819051027297974, -1.9374266862869263, -0.056038375943899155, 0.42744016647338867, 0.6377061605453491, 0.7664955258369446, -0.16767026484012604, -0.155320942401886, -0.2474367916584015, -0.5119990706443787, -1.5336887836456299, -0.2452564388513565, 0.12462735921144485, 1.5975403785705566, -0.7542365789413452, 0.7124210000038147, -1.127308964729309, -0.07949911057949066, -0.04349694401025772, -0.0045369635336101055, -1.1374775171279907, 2.08929181098938, -0.40721777081489563, 1.1673696041107178, -1.3456398248672485, -0.11669578403234482, 0.5824042558670044, -0.6843990087509155, 0.2683276832103729, -0.4299837052822113, 1.158340573310852, -0.4601828455924988, -0.8077630996704102, 0.4799143075942993, -1.0798356533050537, -0.19494962692260742, 1.0685213804244995, 1.2232438325881958, 0.05325273796916008, -0.138891339302063, -1.1175968647003174, -0.8989524245262146, 2.079329013824463, 0.5762816667556763, -0.030848078429698944, -1.5199065208435059, -1.335226058959961, 0.33815908432006836, 0.4788585603237152, -0.9548943638801575, -0.5769034624099731, -0.808547854423523, -1.4077444076538086, 1.4109570980072021, 1.5129157304763794, 0.6946156620979309, 0.15448857843875885, 0.11843395978212357, -0.7419386506080627, 0.08161202818155289, 0.46725520491600037, 2.2995550632476807, -0.3789057731628418, 2.5613980293273926, -0.682462751865387, -1.1301904916763306, -1.6301798820495605, -0.6849642395973206, -0.12928906083106995, -0.8303333520889282, -0.4845203757286072, -2.4350249767303467, 0.20694294571876526, -0.6556233167648315, -1.0776177644729614, 1.0735816955566406, 0.5069651007652283, -0.1752222627401352, 0.16481243073940277, -0.28670307993888855, 0.9769561290740967, -0.8215579390525818, -1.1995642185211182, 1.2832098007202148, -0.239454448223114, -0.3822243809700012, 0.8939767479896545, -0.537688136100769, 0.18659502267837524, -0.8584031462669373, 1.062462329864502, -0.34941890835762024, -0.5014345049858093, 0.7295522689819336, -0.28687116503715515, -1.1210477352142334, 0.11100751161575317, 0.2861350476741791, 1.7511179447174072, -0.03693709149956703, -1.7845743894577026, 0.6523869037628174, -1.276269793510437, 1.9093222618103027, -1.205817461013794, 1.5380418300628662, -0.7878907322883606, 0.2737028896808624, -0.2949150800704956, -0.09381074458360672, 0.14974811673164368, -0.9626843333244324, -0.09373245388269424, -0.003721021581441164, -0.5187119245529175, -0.01929299533367157, -0.11932162195444107, 1.0730749368667603, -1.8715410232543945, -0.5434589385986328, -1.1529123783111572, -0.1871507167816162, 1.8193339109420776, -1.1980209350585938, -1.585845947265625, 0.8623348474502563, -1.6337740421295166, 1.1093518733978271, -0.4878438711166382, -1.209484338760376, -0.5015442967414856, -1.000457525253296, -0.09863613545894623, 0.8188449144363403, -2.049149990081787, -1.312753438949585, -0.018730180338025093, -0.9958374500274658, 0.9629855155944824, 1.1336299180984497, 0.917701780796051, 0.9351887106895447, -0.2185034155845642, -0.05088500306010246, 0.5468002557754517, -2.2410030364990234, 2.340775489807129, 0.17290931940078735, -0.5764939785003662, 0.07697796821594238, 0.7794072031974792, -0.0935378223657608, 0.8148559331893921, -0.8386563658714294, 1.3895201683044434, 0.32585805654525757, -0.022557256743311882, -1.314469575881958, -0.7562332153320312, 0.6724520921707153, 1.3060656785964966, -1.2299323081970215, 0.32914379239082336, 1.2675164937973022, -0.768443763256073, 1.4463121891021729, -0.43289801478385925, -0.6968051195144653, -0.5259982943534851, -1.5044450759887695, 1.3356943130493164, 0.1762467920780182, 0.12340530753135681, 0.19446119666099548, 0.268449991941452, 0.2362646758556366, 0.34542641043663025, 0.24229487776756287, 1.7082880735397339, -0.7900218963623047, -2.4760539531707764, 0.4422963857650757, 1.3060678243637085, -0.8390821218490601, -0.5273811221122742, -0.6535137891769409, 1.0684778690338135, -0.5681430101394653, 0.3811489939689636, 0.5918372869491577, 0.7820127606391907, -0.04356608912348747, -1.7057995796203613, -0.43076977133750916, 1.4235783815383911, -0.053642433136701584, 0.46735307574272156, 0.1587471216917038, 0.639969527721405, 0.5782848596572876, 0.41730064153671265, -1.3162440061569214, 3.6737515926361084, -1.04462468624115, 2.336818218231201, 0.23925183713436127, 0.998841404914856, 0.4314610958099365, -0.5301316976547241, 1.5956778526306152, 0.6502588391304016, -1.0042908191680908, -0.20078244805335999, 0.6795439720153809, 0.5694745182991028, -0.25972485542297363, -0.2716209888458252, -0.3874044716358185, 0.3245723247528076, -0.20228810608386993, 0.3491157591342926, 0.07984817028045654, 0.5340720415115356, 0.30199751257896423, 0.8185916543006897, 0.2918727993965149, 0.9228659272193909, 0.1938815861940384, 0.6653213500976562, -2.2322468757629395, 2.2431817054748535, 0.7853939533233643, 2.291856288909912, -2.6338236331939697, -0.0760449767112732, 0.5074542164802551, 1.1479498147964478, 0.002231599995866418, 0.6003229022026062, 0.0207452941685915, 0.07519196718931198, -0.21884895861148834, -0.5134480595588684, -0.23376834392547607, -0.05598427727818489, 0.6883555054664612, 0.07898487895727158, -0.13226298987865448, 2.0505385398864746, 1.4501099586486816, 0.21037347614765167, 0.2925003170967102, 0.4798681139945984, -0.5963044166564941, 1.4424294233322144, -0.23977287113666534, -0.2979627847671509, -0.12278441339731216, 0.5561114549636841, -0.9775813221931458, -0.5434926748275757, 0.06240034103393555, -0.49858880043029785, -1.1492669582366943, -0.7869989275932312, 0.28478896617889404, -0.3476063013076782, 0.6862442493438721, 1.2629603147506714, 0.18574745953083038, -1.0133658647537231, 0.7010367512702942, 0.5294221639633179, -1.2374457120895386, 0.06001027300953865, 0.6236050128936768, 1.7136229276657104, -0.17685547471046448, -0.8188803195953369, 1.2501307725906372, 1.0051636695861816, 1.36894690990448, -0.37177619338035583, 0.8603037595748901, -1.5848479270935059, -0.8604068160057068, -1.02841055393219, 1.0135459899902344, -0.07152209430932999, 0.2527036666870117, 1.4063700437545776, 0.7870162725448608, -0.3300701975822449, -1.1102724075317383, 2.2672994136810303, 0.9135488271713257, 1.8094103336334229, 0.7602104544639587, -0.9299808144569397, 0.7675650715827942, 0.8438974022865295, -0.4977574348449707, -1.3910633325576782, 0.12292339652776718, -0.23062750697135925, -0.9765772819519043, 0.26155951619148254, -0.2416711449623108, 0.36732015013694763, 0.46018943190574646, 0.8244636654853821, 0.7797607183456421, -1.3507189750671387, 0.1819465607404709, 0.45717117190361023, 0.06853897124528885, -2.4513723850250244, -1.270949125289917, 0.7928033471107483, 0.21714738011360168, 0.9589920043945312, 0.8880978226661682, -1.249448537826538, -0.12270363420248032, -0.6160541772842407, 0.14554041624069214, -2.4698736667633057, 0.5447353720664978, -0.978635847568512, 0.12255322933197021, -0.36454564332962036, 0.39018702507019043, 1.3312911987304688, 0.6036986112594604, -0.4771014451980591, 0.7982426285743713, 0.8908302187919617, 0.019972294569015503, -0.6031821370124817, -1.7322818040847778, 0.037703514099121094, -0.04230082035064697, -1.0010018348693848, 1.823323130607605, -1.8119112253189087, -0.3887537717819214, 0.08001534640789032, 0.07924778014421463, 1.315294623374939, 0.26153886318206787, -0.13805325329303741, -0.15718118846416473, -0.13422460854053497, 1.416357159614563, 0.8415672779083252, -0.06964888423681259, -1.4189271926879883, -0.7315441370010376, -0.20722803473472595, -1.229791283607483, -0.7889758944511414, -0.4908737540245056, 0.11573639512062073, -0.4979054629802704, -0.7639247179031372, 0.25678765773773193, -1.1848057508468628, 2.3486435413360596, 0.07307905703783035, 0.38345977663993835, -1.27255380153656, 0.16073887050151825, 0.7495499849319458, -0.18379725515842438, -1.4780572652816772, -0.775632381439209, -1.270385980606079, 0.502855658531189, -0.3187233507633209, 1.41754150390625, -0.9750891327857971, 0.36793121695518494, -1.2677035331726074, 0.7494703531265259, -0.6130926609039307, 1.0269798040390015, 0.16682206094264984, 1.2701297998428345, -0.189788356423378, 1.4571974277496338, -0.259655237197876, -1.4397352933883667, -1.1111599206924438, -0.1566317230463028, 0.9363961219787598, 1.5969380140304565, 0.9231526255607605, 0.5870881080627441, 1.2952888011932373, -0.773546576499939, 0.5366474986076355, -0.0754498541355133, 0.7982910871505737, -0.40311771631240845, 1.4107362031936646, 1.3407142162322998, 1.4153169393539429, 0.548958957195282, -0.9829663634300232, 0.4669360816478729, -1.4613001346588135, 0.30908113718032837, 1.0650923252105713, -0.2913859784603119, 0.2711656391620636, 0.7187421321868896, -0.21846549212932587, 0.7250267863273621, -0.9741449356079102, 1.0808906555175781, 0.06326127052307129, 0.9863758087158203, -0.6230910420417786, -0.7764421105384827, 1.2902750968933105, -1.1307957172393799, -0.7453846335411072, -0.2620645761489868, 0.023752989247441292, 0.9144415855407715, 0.2826217710971832, 1.3904600143432617, 0.9530930519104004, -1.412315011024475, 0.02689194306731224, 1.604673147201538, -1.4835829734802246, 0.8966420292854309, -0.32739198207855225, 1.2441788911819458, 0.7311781644821167, 0.004459458403289318, -0.1788874864578247, 1.295315146446228, -0.11820317059755325, 0.21517957746982574, 1.1106696128845215, 0.6536160707473755, 0.21766939759254456, -0.17091822624206543, -2.163475751876831, 0.07682181149721146, -0.7173066735267639, -1.370871901512146, -0.19838067889213562, -0.4229986369609833, 0.283109575510025, 0.5211254954338074, -0.5563114881515503, 0.640447735786438, -1.5703728199005127, -0.5166107416152954, 0.2955002784729004, -0.3421171307563782, 1.2317020893096924, 0.06280237436294556, -0.5667895674705505, -0.5517247319221497, -0.6028580069541931, 0.5310319662094116, -0.022019291296601295, -0.5927911400794983, 2.47328519821167, 0.5448393225669861, -0.9754891395568848, -1.3513048887252808, -0.9707645773887634, 0.16731448471546173, -0.5533390045166016, -0.6041549444198608, 0.407719224691391, 0.49439623951911926, 0.7100433111190796, 0.37007200717926025, 0.5994689464569092, -0.8576645851135254, 0.9182778000831604, -0.29588717222213745, 0.46340951323509216, -1.553897500038147, -2.4943394660949707, 0.6516563296318054, -0.7878957390785217, 0.4741089642047882, -0.8277125358581543, -0.20747846364974976, 0.5684258937835693, -0.5316244959831238, -0.48980650305747986, 0.1905922144651413, 0.9824312925338745, 0.8332682847976685, 0.2877182364463806, -0.05639726668596268, -0.5890053510665894, 0.855527400970459, -1.5061177015304565, 1.2436069250106812, -0.5304905772209167, 1.4433451890945435, 1.3828331232070923, 0.14383888244628906, -1.4644054174423218, 1.0306224822998047, 1.1291226148605347, -0.3243328034877777, 1.1517881155014038, -0.8547847867012024, 0.7504647374153137, -1.7547754049301147, -0.08191923797130585, 1.038068175315857, -0.041076403111219406, 0.5999711751937866, -0.4698258936405182, -0.4679153263568878, -0.9513369798660278, 0.6074690818786621, -0.2762940526008606, 1.367124080657959, -0.8126654624938965, -0.1106005534529686, -0.11799086630344391, -0.5566234588623047, -0.40771397948265076, -1.308271884918213, 1.170082688331604, -1.3648127317428589, -0.26598647236824036, -0.7122929096221924, -0.5797160863876343, 0.22422903776168823, -0.4537004232406616, -1.8840612173080444, -0.4114074110984802, -0.4816023111343384, 0.12626437842845917, 0.7607876658439636, 2.259258270263672, 0.3308235704898834, -0.33162015676498413, -0.23643499612808228, 1.2636886835098267, -1.9621176719665527, 0.3868870139122009, -0.18622466921806335, -1.278501272201538, -1.0189967155456543, -0.40139225125312805, -0.017008798196911812, 2.161867380142212, -0.3058689534664154, 0.7196232080459595, 0.47980859875679016, -0.7053108811378479, 1.7766209840774536, 2.4893510341644287, 1.1989189386367798, -1.0620166063308716, 0.2687249481678009, 1.1247339248657227, -1.9386001825332642, 1.4358407258987427, 0.5430940389633179, 0.09647853672504425, -0.014841767027974129, -1.5541176795959473], [1.7190752029418945, -0.7576301097869873, 1.5561389923095703, -0.1848706156015396, 1.7105778455734253, -0.2780018448829651, -1.2440329790115356, 2.063558340072632, 0.37932077050209045, 0.94498211145401, -1.1894252300262451, 0.06879077106714249, -0.7234466075897217, -0.4335184693336487, 0.7181171178817749, 2.8672902584075928, -1.584848403930664, -0.3221296966075897, 0.3865828514099121, 0.06004077196121216, -0.7966383695602417, -0.8847243189811707, -0.5258080959320068, 0.06834857165813446, -0.9205175638198853, -1.1240960359573364, 1.5241326093673706, -0.41477885842323303, 0.2249569445848465, 1.0020253658294678, -2.407412052154541, -0.7392876148223877, -0.6200862526893616, -0.1675851047039032, 1.1290746927261353, -0.5480465888977051, 0.09845228493213654, -0.3230660557746887, -1.0862678289413452, -0.12759412825107574, 0.9852273464202881, 0.07187917083501816, 1.1438015699386597, 0.1783490628004074, -1.400695562362671, 1.0800485610961914, 0.042497023940086365, 1.287290334701538, 0.1951114684343338, 0.5915694236755371, 0.008913218975067139, 0.25487399101257324, -0.2238095998764038, -0.7601404786109924, -0.20419573783874512, -1.4328832626342773, -1.2187591791152954, -0.13789476454257965, 1.53133225440979, -0.2425985336303711, -0.33504998683929443, -1.105898380279541, -2.3699634075164795, -0.772432804107666, 0.22713689506053925, 1.1122002601623535, -0.29840636253356934, 1.330615758895874, -1.375324010848999, 1.2327905893325806, -0.23947063088417053, 1.6069681644439697, 2.1760971546173096, 2.408039093017578, 0.7669594287872314, -1.1294571161270142, 0.33114930987358093, -1.0805507898330688, -0.4318731725215912, -0.797653079032898, -0.32927224040031433, 0.8473027944564819, -1.5577070713043213, 0.5231571197509766, 2.4159369468688965, -0.9619498252868652, -0.020596373826265335, 1.1500519514083862, -0.23852653801441193, -0.4308493137359619, 2.009091854095459, 0.09453198313713074, -0.7907021045684814, 1.4440639019012451, -1.614586591720581, -0.162001371383667, 0.051938410848379135, 0.40142232179641724, 0.3083640933036804, 0.9949273467063904, 0.2889237105846405, -0.8796060681343079, 0.5886751413345337, 0.18675734102725983, 0.3455609679222107, 0.6861407160758972, -1.2165651321411133, -0.6650339961051941, 0.3067399561405182, -0.8546951413154602, 1.8834953308105469, -0.7291730046272278, 1.2819417715072632, 1.6751503944396973, -1.1193333864212036, -0.5755713582038879, -0.7287359237670898, -0.6476840972900391, -0.35598117113113403, -0.47305312752723694, 0.6414878368377686, 1.348379373550415, 0.2626877427101135, 1.84639310836792, 0.3077210485935211, -0.7485102415084839, -0.19534353911876678, 0.11930767446756363, 0.24134160578250885, 0.742811381816864, 0.45510798692703247, -0.1580202579498291, -0.5553134083747864, -1.2904661893844604, 0.23612801730632782, -0.6750380992889404, -2.826192617416382, -0.45278027653694153, 2.118321180343628, -1.1809383630752563, 0.34315672516822815, 0.17957136034965515, -1.2837986946105957, 0.06022794917225838, -0.6992911100387573, -0.9699043035507202, -0.3067813217639923, 1.3919198513031006, 1.1164132356643677, -0.6367223858833313, 0.04271621257066727, 1.9173263311386108, 0.1255432814359665, -2.3693339824676514, 0.28865790367126465, -0.23906365036964417, 0.4388156831264496, 0.832283079624176, -1.319153904914856, -0.5016701221466064, 0.2884683609008789, 1.0966508388519287, 0.9487335681915283, 0.39004117250442505, 0.21742865443229675, -0.4843044877052307, 0.06793619692325592, -0.8503576517105103, -0.02708398923277855, 0.8738315105438232, 1.133342981338501, 0.11938226222991943, -0.20359666645526886, 0.3568839430809021, 0.44946396350860596, -0.8755762577056885, -0.39007505774497986, 0.8468305468559265, -0.1810106486082077, 0.07686934620141983, 0.43855372071266174, 0.21576854586601257, -0.1628025621175766, -0.058320026844739914, 0.6487125754356384, -0.3909374177455902, -1.2954257726669312, -0.17417246103286743, 0.30330368876457214, 1.2925851345062256, 0.5584257245063782, -1.9163509607315063, -0.429625928401947, -0.23292580246925354, 0.44288700819015503, -0.6654269099235535, 0.4253969192504883, -1.1569104194641113, -0.173285111784935, -0.055949270725250244, -0.45566076040267944, -1.4025150537490845, 0.13749383389949799, -0.6916940212249756, 1.3917855024337769, 0.4590907394886017, -0.42399463057518005, -0.6840639114379883, 0.9187433123588562, 0.788931667804718, 2.191387891769409, -0.2084168940782547, -0.36667922139167786, -1.0886541604995728, -0.5994493365287781, 0.27489781379699707, -1.2655616998672485, -0.6763235926628113, 1.7083274126052856, -0.022928504273295403, -0.13500899076461792, -1.6881245374679565, 0.6691079139709473, -0.9512373208999634, 2.149689197540283, -0.49909672141075134, 0.053544897586107254, -1.0255573987960815, -0.2981737554073334, 0.009795701131224632, -0.30107781291007996, -1.063444972038269, -0.8094069361686707, -1.5864907503128052, 0.7839874029159546, 1.65937077999115, 0.15795785188674927, -0.8475328087806702, -0.3993443548679352, -0.7656976580619812, -1.330496907234192, 0.20507219433784485, 0.11944152414798737, 0.4071531295776367, 1.170771837234497, 0.7028201818466187, 0.9825494885444641, -0.3570724427700043, 0.5137879848480225, 1.295701503753662, -0.5500614643096924, -0.05430828034877777, -0.09500803053379059, 0.39849695563316345, 1.230790376663208, -0.3736015856266022, -0.7662920355796814, -0.4763457775115967, 1.2391929626464844, -1.6993434429168701, -0.7483858466148376, -0.39729875326156616, -1.1361340284347534, 0.7878902554512024, 1.0600937604904175, 0.2714245319366455, 0.2972211241722107, 1.9459596872329712, 0.717114269733429, -1.4929457902908325, -0.3426735997200012, 0.3583807647228241, -0.5168792009353638, 0.6763659715652466, 1.0585957765579224, 0.8244444727897644, -0.7214047312736511, 0.5432040095329285, 1.284245491027832, -0.3263133466243744, -0.1368665248155594, 0.6188595294952393, 0.020823432132601738, 0.07463200390338898, 0.055215153843164444, 0.09885963797569275, -1.9324008226394653, 0.6640192866325378, -0.2353885918855667, 1.32417893409729, -1.360886812210083, 2.6544618606567383, 0.3739883005619049, -0.5673415064811707, 1.0115290880203247, 0.21671606600284576, 1.6333396434783936, 0.2603847086429596, -0.8950830101966858, -0.3264191448688507, 0.11302206665277481, -0.9605692028999329, -1.145688533782959, 1.3919705152511597, 0.01861870475113392, -0.9669780135154724, -0.22757959365844727, -1.4175395965576172, -1.0121968984603882, 0.558418333530426, 0.25267624855041504, 0.6438395380973816, -0.9326446652412415, -1.2349172830581665, 0.4670563042163849, 0.8822600245475769, 0.22443974018096924, 1.1011215448379517, -1.1961400508880615, 0.954466700553894, 0.03757946565747261, 0.12282399088144302, 1.1147301197052002, 1.32355535030365, 0.9805838465690613, 0.40333092212677, 1.2696621417999268, 0.08106180280447006, 1.333526611328125, -0.19645744562149048, 0.05765259265899658, -0.8247660994529724, -0.33978867530822754, 0.14062897861003876, -0.7944046258926392, -0.6664648652076721, 1.4891434907913208, 0.2350482940673828, 0.6071981191635132, -0.09911133348941803, -2.641838788986206, 0.984938383102417, 1.7690638303756714, 0.8686102032661438, 0.054843466728925705, -0.9450642466545105, -1.2120742797851562, 0.4210160970687866, 0.2681524157524109, 1.438970685005188, 1.016234278678894, 0.07046579569578171, 0.6974981427192688, 0.22194132208824158, -0.8242951035499573, -0.7850839495658875, -0.8745124340057373, -0.8321242332458496, -1.2528355121612549, -0.09491650015115738, -1.0157184600830078, 2.434654712677002, 0.08264075219631195, -0.4418482780456543, 0.845268189907074, -0.7815672755241394, 1.427164077758789, -1.589032769203186, -0.5701972842216492, 0.5503079891204834, -0.595073401927948, 0.6657289266586304, -2.405747652053833, 0.03892442584037781, 0.9457436203956604, -0.4613703191280365, 0.7826873064041138, -0.704638659954071, -0.9749639630317688, 2.5968940258026123, 0.14788329601287842, -0.10916324704885483, 1.8612217903137207, 0.2407648265361786, 1.2591164112091064, -0.6940690875053406, -1.1059414148330688, 1.068469524383545, 1.3889845609664917, 1.0579369068145752, -1.411631464958191, -0.8245639801025391, -0.16709336638450623, 0.1243039071559906, -1.0191458463668823, 1.406886339187622, 0.9652644991874695, -0.018910882994532585, -0.47754359245300293, 0.28059786558151245, -1.6272350549697876, -0.1215561255812645, -0.1250867396593094, 1.515040397644043, -0.028151068836450577, -0.3241780698299408, -0.5938573479652405, 0.7730191946029663, 0.6689319610595703, -0.9305031299591064, 0.3236398994922638, 0.3298453092575073, 0.0225015040487051, 0.7712369561195374, -0.4628407657146454, 0.2870069146156311, 1.7815613746643066, 0.822763979434967, 1.4161875247955322, -0.24762596189975739, 0.6745560765266418, 1.411144733428955, -0.5603938698768616, 0.5425670146942139, -1.0615822076797485, 0.13801681995391846, -0.9914199113845825, 0.7151121497154236, -0.6466354727745056, -1.7615976333618164, -1.45018470287323, -0.7192772626876831, -1.7158668041229248, 0.5576186180114746, -0.7566083669662476, -0.5949252843856812, 0.7731032967567444, -1.025887131690979, -1.5345072746276855, 0.6805869340896606, 1.3711165189743042, 0.12101724743843079, -1.3927520513534546, 0.9895105361938477, -0.5425228476524353, -0.32187360525131226, 0.6570711135864258, 0.16996431350708008, 1.151544213294983, 0.31585368514060974, 3.1709823608398438, -1.0081584453582764, -0.23414278030395508, 0.02981737069785595, 0.6860515475273132, 0.4506894052028656, 0.5749047994613647, -0.49325132369995117, -0.7415721416473389, 0.771020233631134, 0.3741450309753418, -2.077263355255127, 0.7772891521453857, 0.7596520185470581, -2.107150077819824, 0.01013902761042118, 1.4254229068756104, 0.24425634741783142, -0.9648193717002869, -0.4101470410823822, -0.7075136303901672, -0.890370786190033, 0.43533313274383545, -0.84428870677948, -0.7525481581687927, -0.7859960198402405, -0.008572929538786411, -0.30556192994117737, 0.1550159603357315, 0.8087039589881897, 1.741059422492981, -0.7854663729667664, 0.8588377833366394, 2.1715221405029297, -0.26440194249153137, 0.028568720445036888, 0.6641527414321899, -1.190278172492981, -1.2369372844696045, 1.2079854011535645, -0.751502513885498, -1.5516923666000366, 1.4813207387924194, 0.5970155596733093, -0.40251508355140686, -0.43589702248573303, 1.4224339723587036, 0.030529651790857315, 0.6690637469291687, 0.6695167422294617, 0.26913607120513916, -1.144747257232666, -0.42407333850860596, 0.8165475726127625, 0.40422946214675903, -0.18271024525165558, -0.5265587568283081, 1.0542727708816528, 0.18497975170612335, 1.0443124771118164, 0.3692237436771393, -0.5804541707038879, -0.5386366844177246, -1.4248615503311157, 1.2723045349121094, 2.6314549446105957, -0.11263900250196457, 0.5120846629142761, 0.5381720662117004, 0.25920867919921875, 0.06415712833404541, 1.215537190437317, -0.005033314693719149, -0.11558520793914795, 0.6663855314254761, 0.09290839731693268, 0.070949025452137, -2.0637471675872803, -1.8857800960540771, -0.023805348202586174, 0.8732540607452393, -0.6046544909477234, -0.4122077524662018, -0.48110392689704895, 0.48269394040107727, 1.2613815069198608, 0.8965204358100891, 1.2925994396209717, 1.5301250219345093, 1.9140783548355103, -0.7915128469467163, -1.5240156650543213, -0.328658789396286, 0.36606091260910034, -1.0887665748596191, -1.8361443281173706, -0.4866218864917755, 0.28248700499534607, 0.03878588229417801, -0.3858223557472229, -0.16321022808551788, -0.13870546221733093, 0.14759601652622223, 0.38115012645721436, -0.14496998488903046, 1.0172252655029297, -1.0458954572677612, -0.35755789279937744, -0.6788167953491211, 1.125184416770935, 0.9019858837127686, 1.9487210512161255, 1.9553453922271729, -1.1132307052612305, -0.27844974398612976, -1.0609736442565918, 0.37668275833129883, 1.1011288166046143, 0.8014376163482666, 0.15918882191181183, 0.9319593906402588, -0.1625339835882187, 1.3699021339416504, -0.09935528039932251, -0.3302081525325775, 0.4529593288898468, 0.13194893300533295, 0.2688794732093811, 0.475813627243042, 1.0166774988174438, -0.10652625560760498, -2.228987693786621, 0.4910328984260559, 0.4666312038898468, 0.07812529057264328, 1.0187954902648926, 0.09107107669115067, -0.9055551886558533, -1.6291556358337402, 0.4045718014240265, 1.0168302059173584, -2.1349527835845947, 0.8016631603240967, 0.7492204904556274, 1.6114821434020996, -0.05843699350953102, -0.8199542760848999, 0.43132123351097107, 0.14213180541992188, 1.7344642877578735, 0.11569337546825409, 2.1116464138031006, -1.4396380186080933, -1.070570707321167, -0.8184475302696228, -1.4519789218902588, 1.0354387760162354, -0.2063473016023636, -1.0150710344314575, 0.894519567489624, 0.3855593502521515, -0.3626250922679901, 0.3810043931007385, -0.999504566192627, 0.3412843644618988, -0.4265810549259186, -1.1468479633331299, 2.239797592163086, -0.7936710715293884, -1.0884572267532349, -2.4598031044006348, 1.2156968116760254, -1.0959410667419434, 0.3870024085044861, -1.0268605947494507, 0.1931336522102356, 0.025707010179758072, -1.3111293315887451, 0.829336404800415, 0.2839735150337219, -0.35657548904418945, 0.416985422372818, -0.8284708857536316, 0.05923919379711151, 0.6631579399108887, 1.2219923734664917, -0.47779640555381775, -0.09679289907217026, -0.3963678777217865, 0.3322720527648926, -0.18261995911598206, -0.09211810678243637, 0.62204909324646, 0.9200994372367859, -0.6548497676849365, 0.00943723600357771, 2.2743773460388184, -0.428104430437088, 1.4380481243133545, -1.0757336616516113, 0.07725253701210022, 0.16486142575740814, 0.7624019384384155, -0.5751859545707703, -0.2966310381889343, 0.45620793104171753, 1.3601176738739014, -0.9344179034233093, -0.7927622199058533, -2.016775131225586, 2.4521727561950684, 0.14683851599693298, -0.9679055213928223, 0.07951570302248001, -0.04176883026957512, -0.10431141406297684, -0.21259726583957672, -1.1342601776123047, -0.10803654789924622, -0.10692976415157318, 1.8058453798294067, 0.7959898114204407, -0.9257637858390808, -0.9792183041572571, 0.21503174304962158, -0.5718541145324707, -1.9820419549942017, 0.004294632002711296, -0.31611254811286926, -0.43723517656326294, 0.2999327480792999, -2.420009136199951, 0.7130074501037598, -0.1359727382659912, -0.18430277705192566, 0.7985629439353943, -0.6147056818008423, -0.09535332024097443, 0.5897821187973022, 2.35294771194458, -1.5877858400344849, 0.9375067949295044, 1.0519031286239624, 1.827197551727295, -0.6556649804115295, -0.07784511893987656, 1.6691292524337769, -0.24844928085803986, -0.1393229067325592, -0.8145198225975037, -0.020969903096556664, -0.32435303926467896, -0.2990453839302063, -0.8245222568511963, -0.2381407767534256, -0.180450901389122, -0.9474138021469116, 0.72392338514328, -0.635051965713501, -1.801324486732483, -0.3199370503425598, 0.7007043361663818, -0.05145561695098877, 0.27574366331100464, -0.714225172996521, 0.5187893509864807, -0.956096887588501, 1.31028413772583, 2.410754680633545, 0.7304751873016357, 0.4943508207798004, 0.8738729357719421, 0.13131654262542725, 0.2631379961967468, -0.11217635124921799, -0.7307342886924744, 0.8892745971679688, 0.9798721671104431, 1.2453371286392212, 0.9065426588058472, -1.4346729516983032, 0.20883941650390625, 0.3455275595188141, -0.9334984421730042, 0.7929637432098389, -0.7776868939399719, -0.020095327869057655, -0.4231144189834595, -0.12628266215324402, 0.989080548286438, 1.0663042068481445, 0.773722231388092, 0.3385652005672455, 0.2781781852245331, 0.8760191202163696, -0.6349814534187317, 0.7714172601699829, -1.2216233015060425, 1.0342439413070679, 0.7327982783317566, 0.6023612022399902, -2.425442695617676, 1.3957295417785645, 0.15448302030563354, -0.9897884726524353, -0.25438132882118225, 0.8350847363471985, 1.2587172985076904, -0.3919863700866699, 0.028166137635707855, -0.49830687046051025, -0.2862885296344757, 0.1021491140127182, 0.31172341108322144, 0.17274552583694458, 0.6739091873168945, 0.4723687767982483, 1.2296521663665771, -0.2915017008781433, -0.5254523754119873, -0.7081851363182068, 0.6248892545700073, 1.4262717962265015, 0.2909340262413025, -3.4565629959106445, 0.785789430141449, -0.043636538088321686, -0.3258800804615021, -0.3054945170879364, 1.0939717292785645, -0.2809513211250305, 0.31688353419303894, 0.03954550251364708, 1.2308100461959839, -0.9584965109825134, -1.3394533395767212, -0.7578352093696594, -0.07411590218544006, -0.20479519665241241, -1.357352614402771, -0.7148149013519287, -0.4835478961467743, -1.1717180013656616, -0.29902833700180054, -0.4134003520011902, -1.230272650718689, 1.7640613317489624, -1.0800997018814087, 1.6849803924560547, -3.0640103816986084, 0.12402356415987015, -3.1571133136749268, -0.17799656093120575, 0.18410789966583252, -0.32232341170310974, -0.12721207737922668, -1.1084779500961304, -0.9398832321166992, -0.663943350315094, 0.9170634746551514, -0.18268461525440216, 2.404731273651123, -1.2791978120803833, 2.3533120155334473, 0.6814747452735901, -0.08914974331855774, 1.7706629037857056, 0.15732096135616302, 1.5363845825195312, 0.493560254573822, -0.2722497880458832, -0.6150481104850769, -0.7497890591621399, 0.8043853044509888, 2.4063618183135986, 1.1548782587051392, -0.04164633899927139, 1.2124414443969727, 1.4833365678787231, -0.6856052875518799, 1.0872538089752197, -0.5353164672851562, 1.3779605627059937, 0.36386528611183167, -0.6324434280395508, 1.7063506841659546, -1.7192230224609375, 0.996768057346344, 1.8863086700439453, 1.3354789018630981, -1.7114242315292358, 0.8569813370704651, 0.5504949688911438, -0.13501641154289246, 0.42123159766197205, -1.2151180505752563, -0.06717649102210999, 1.4200778007507324, -1.2972711324691772, 0.9834616184234619, 0.7031345367431641, -0.05139319226145744, -0.5679023265838623, -1.3268147706985474, -0.2949275076389313, 1.1446412801742554, -1.541193962097168, 0.18075747787952423, 0.4920787513256073, -1.946323037147522, 0.4217931926250458, -0.4882209599018097, -1.8681668043136597, 1.0369644165039062, 0.9032266736030579, 0.71419358253479, -0.5080370903015137, 2.0332586765289307, -0.3309788703918457, 1.2365258932113647, 0.5674504041671753, -1.9011046886444092, 0.213381826877594, 1.1653770208358765, 0.5056928992271423, -0.9376643300056458, 0.7832084894180298, 0.013179225847125053, -0.4298263192176819, 0.39413920044898987, 0.9197533130645752, -1.718790054321289, 1.2748908996582031, -0.6310462951660156, 1.4002958536148071, 1.3118890523910522, 0.19982890784740448, 0.896192193031311, 0.09840214997529984, -0.09647953510284424, 0.3329910337924957, 0.5894656181335449, -0.9965479969978333, 0.13700827956199646, -0.23063592612743378, 0.07078398764133453, -1.5689027309417725, 0.6689156293869019, -1.2152273654937744, -0.35826218128204346, -0.3002150058746338, 2.260483741760254, 0.4269881546497345, -0.9674636125564575, -0.7952966690063477, -1.4195002317428589, 0.7256695032119751, 1.218549132347107, -0.11619838327169418, -1.4985220432281494, -0.3975987732410431, 0.8393891453742981, -0.4846286475658417, -0.6030272841453552, 1.644795536994934, -0.443155974149704, 1.4603887796401978, 0.035479262471199036, 1.572271466255188, 1.4027645587921143, 1.213014841079712, 2.0916695594787598, -2.2519078254699707, -1.182672142982483, 0.20808635652065277, 0.5729490518569946, -2.305424213409424, -1.539465069770813, -0.675015926361084, 1.0405622720718384, 0.2891804873943329, 0.28648531436920166, 2.590128183364868, -0.687352180480957, 0.1727599799633026, -0.9795306921005249, -1.0043885707855225, 0.6716340780258179, -3.3041605949401855, 0.7085877656936646, 1.7325644493103027, -0.3206862509250641, 1.0693029165267944, 0.0357096828520298, 1.3463170528411865, -0.8666722774505615, -1.0676801204681396, -0.11420129239559174, 1.2878317832946777, -0.8351755738258362, -0.886738657951355, -0.4861663579940796, 0.04691244661808014, -1.3095066547393799, 0.8381665349006653, 1.0602062940597534, 0.37419387698173523, -0.3965778052806854, -0.21198704838752747, -1.1777769327163696, 0.06845638155937195, 0.5469830632209778, 1.1257977485656738, -0.0014050969621166587, 0.17350070178508759, 0.563789427280426, 0.4039762318134308, 0.2668275535106659, -0.9335907697677612, -0.030924363061785698, -1.2490978240966797, 0.6785452961921692, -0.15062309801578522, -0.23715132474899292, -0.18342430889606476, 1.6173800230026245, -0.016757547855377197, 0.26543891429901123, 0.21242782473564148, 1.4002710580825806, -0.9203652739524841, 0.6745657920837402, 0.11089381575584412, -1.0158981084823608, -0.23603582382202148, 0.110948346555233, -1.3929897546768188, -0.17320987582206726, 1.0450668334960938, 1.478163480758667, -1.4114768505096436, -0.47932466864585876, 0.22870591282844543, -0.5337777137756348, -0.9177247881889343, -0.2861217260360718, 0.2831376791000366, -0.6338320970535278, -0.6415305733680725, -0.697794497013092, -0.37946513295173645, -0.17756205797195435, -0.8272179961204529, 0.06545109301805496, -1.3549799919128418, -0.6982210278511047, -1.6403639316558838, -0.06270015239715576, -0.3525274991989136, 0.6487477421760559, -0.3966192901134491, -0.024035153910517693, -1.1198482513427734, -1.2490373849868774, -0.052487485110759735, -0.7228025197982788, -0.3302008807659149, 0.863586962223053, 0.7350131869316101, 0.6248423457145691, 0.21282854676246643, 0.8275285959243774, 0.8274446129798889, -0.6234923601150513], [0.5992450714111328, -0.15289022028446198, 1.5474538803100586, -0.009654941037297249, 2.132425308227539, -2.1058614253997803, -0.7662757039070129, 0.8972680568695068, -1.9277821779251099, 0.9323208332061768, -0.08100532740354538, 0.8820127248764038, 0.7486807107925415, 1.4504890441894531, -0.9174568057060242, -2.3864712715148926, -0.6725347638130188, 1.222908854484558, -0.746235191822052, -0.16519425809383392, 0.20676055550575256, 0.6515044569969177, 1.5770331621170044, 1.0425045490264893, -0.34259140491485596, 0.42811110615730286, 0.27456575632095337, 0.8906510472297668, -0.5328671336174011, 1.606340765953064, 0.3628605008125305, 0.5002110600471497, 0.14182539284229279, 0.09051303565502167, 0.7206474542617798, 1.3956174850463867, 0.4573180079460144, -0.12318231910467148, 0.11894115805625916, 1.8318294286727905, -0.14184565842151642, 0.6656888723373413, -1.1867464780807495, 0.32092469930648804, 0.45706093311309814, -2.2012851238250732, 0.2552117109298706, 0.7473332285881042, -1.0232820510864258, 0.9973653554916382, 0.005607496947050095, 2.7917368412017822, 0.5705450177192688, -1.1304205656051636, 1.2168688774108887, 0.3558976948261261, 2.899280071258545, -0.9078918099403381, 0.16196894645690918, -0.14165742695331573, 0.6318187713623047, 0.925247311592102, 0.6105418801307678, -0.19563432037830353, 0.15855629742145538, -0.35214605927467346, 0.4771570563316345, 0.40746796131134033, -0.28534308075904846, 0.2925785779953003, 0.19739489257335663, -0.31069767475128174, 0.860079824924469, 0.7056297063827515, -0.6306665539741516, -1.8610663414001465, -1.3751237392425537, 0.746480405330658, 1.0905749797821045, 0.17213863134384155, -0.7622566223144531, 2.124309778213501, -0.23429973423480988, -0.11429498344659805, -1.2389209270477295, 1.0336216688156128, -0.441650390625, -0.715222954750061, 0.529370129108429, 1.2013558149337769, 0.3673982620239258, -1.0470554828643799, -0.33184412121772766, -0.3774944841861725, 0.878723680973053, 0.5247358083724976, -0.26793473958969116, 0.08596570789813995, 0.9284179210662842, -0.2817826569080353, -0.2159152328968048, 0.18003274500370026, 2.1528189182281494, 2.0615973472595215, 0.1689184457063675, 1.3537542819976807, -1.19903564453125, 0.5543383955955505, -0.36419835686683655, 0.11184418946504593, -1.6436008214950562, 0.3603416383266449, 1.1405425071716309, 0.6239582300186157, -1.1112107038497925, 0.7049394249916077, 0.5836202502250671, 0.32676857709884644, -0.8873512148857117, -0.22455117106437683, -0.0159535463899374, 1.0135587453842163, -1.618632435798645, 0.12318798899650574, 0.9186615347862244, -0.975829005241394, -1.580005407333374, 2.220796585083008, -0.33770909905433655, -0.5217429399490356, -1.3071116209030151, 0.20563435554504395, 0.753873348236084, -0.21350090205669403, 0.812585175037384, -0.4994092583656311, -0.1186300590634346, -0.28742486238479614, -1.4601812362670898, -0.9485294818878174, 0.1148693710565567, 0.04740895330905914, 1.076873779296875, 1.9142764806747437, 0.513563871383667, -2.5314016342163086, -0.8240318894386292, 0.6698610186576843, -0.09516022354364395, 0.060619380325078964, 1.4278268814086914, -0.1383122056722641, -0.28050652146339417, -0.7215436100959778, -0.28151795268058777, -1.4499632120132446, 1.190909743309021, -0.4612988829612732, -0.3248692452907562, 0.7176153659820557, -0.8712452054023743, -0.06161006912589073, -0.7551819086074829, -1.547126293182373, -0.03841717913746834, -1.6585346460342407, 0.36503466963768005, -1.3605021238327026, -0.004952066577970982, -0.6299843192100525, 0.023578399792313576, -1.1083784103393555, -1.0044277906417847, -1.4528346061706543, 1.6305673122406006, -0.8274168968200684, 0.08570251613855362, 0.8671227097511292, -2.416421890258789, -1.752219557762146, 0.7230563163757324, 0.7028168439865112, 0.4138796329498291, -1.5224884748458862, 1.17747163772583, -1.3434046506881714, 1.3620378971099854, 0.4692073464393616, 0.6848742961883545, 0.12265264987945557, 0.931774377822876, 0.37327250838279724, 0.8785009384155273, 1.0284321308135986, 1.1321653127670288, 0.6624229550361633, -1.0539695024490356, 0.4763941466808319, -0.40928560495376587, 0.5579655170440674, 2.5516116619110107, -1.5432250499725342, -0.02299918606877327, -0.6170274019241333, -0.7299344539642334, 1.0412676334381104, 0.43175652623176575, -0.6337701678276062, -0.40353718400001526, -1.2033507823944092, -0.3183629512786865, 1.5070401430130005, 1.4503123760223389, 0.9556272029876709, -0.852819561958313, -0.2026209980249405, 0.8154544234275818, -1.0663548707962036, 0.6087619066238403, -0.3310929238796234, -0.18554556369781494, 1.4734623432159424, 0.4491613507270813, -0.5452728867530823, 2.497553586959839, 0.9754694104194641, -2.9007925987243652, 0.47334542870521545, 1.6323583126068115, -1.8601003885269165, -1.2193880081176758, 0.18587949872016907, -1.0103545188903809, -0.22788052260875702, 0.4181320369243622, 0.5034149885177612, 0.24278724193572998, 0.06324242800474167, 0.4800552725791931, -1.1829713582992554, -0.44659852981567383, 1.2287933826446533, 0.760650098323822, 0.02809927426278591, 0.6847181916236877, 0.12088172137737274, -0.09888143837451935, 0.12005089968442917, 0.007529244292527437, 0.5130947828292847, 1.4749616384506226, 0.3523717224597931, -0.9464820623397827, -1.4669342041015625, -1.2808945178985596, 0.06268257647752762, -1.701048493385315, -0.9886860251426697, -0.05753445252776146, 1.5265427827835083, -0.1157812625169754, -1.0700491666793823, -0.5912368893623352, 0.12568452954292297, 1.6161518096923828, 0.488614559173584, 0.9021927118301392, 1.4639898538589478, 0.15890124440193176, 0.0740717202425003, -1.796912431716919, 1.2045634984970093, -0.5014359951019287, -1.1502612829208374, -2.2447240352630615, 0.5646876692771912, -0.5185551047325134, -0.4529307782649994, -1.1197694540023804, 0.1694844365119934, -0.16746734082698822, -0.23800969123840332, -0.3125288784503937, 1.0613279342651367, 0.8652249574661255, 0.7196421027183533, -0.4935920834541321, -2.5849080085754395, -0.24517680704593658, -0.8200402855873108, 0.28857120871543884, -0.2885856032371521, -1.4040168523788452, -1.3664029836654663, 1.3718769550323486, -0.6347402334213257, 0.8309693932533264, -0.49888283014297485, 0.056585270911455154, 1.8050920963287354, 0.2649437487125397, 0.3434080183506012, 0.9385610818862915, -0.22714796662330627, -2.395932912826538, 0.13726133108139038, -0.6198161840438843, 1.2074086666107178, -0.9438591003417969, 0.38165855407714844, -0.7709546089172363, -0.3225954473018646, 0.00225092307664454, -1.6703068017959595, -0.09688408672809601, -1.419079065322876, -0.5370932221412659, -0.6004882454872131, -0.4368394911289215, 0.7305861115455627, 2.567497491836548, -0.6990485191345215, 1.5540275573730469, -0.8174817562103271, 0.015086264349520206, -0.819820761680603, 0.8746203184127808, -1.4035197496414185, -1.234082579612732, -0.8401569128036499, 0.2886868119239807, -0.33797186613082886, -0.06783512234687805, 0.7459997534751892, 0.37685397267341614, -0.13884547352790833, -0.5013725161552429, -1.2706761360168457, 0.03314361348748207, 1.0330495834350586, 0.06055694818496704, 0.6251688003540039, 0.95235675573349, -0.373597651720047, -0.989924430847168, -0.058831073343753815, 0.6435036659240723, -0.40756136178970337, 0.11425434798002243, 0.04791533201932907, -0.9127292037010193, -0.5365790724754333, 0.46803024411201477, 1.9275892972946167, -2.0141892433166504, 1.423547625541687, -1.6771259307861328, 1.3397177457809448, 0.22516684234142303, -1.2567847967147827, 0.6265572905540466, -0.18941070139408112, -0.4149099886417389, -0.6871014833450317, -0.2172776609659195, 0.07815160602331161, 1.8196449279785156, -0.8578728437423706, -0.07881730049848557, 1.7900534868240356, -0.5754793286323547, 0.7096680998802185, -0.11606091260910034, -0.7779379487037659, 0.8404251933097839, 1.3435434103012085, -0.9814277291297913, 0.008672586642205715, -1.1508461236953735, -0.045904021710157394, -0.5331099629402161, 1.7235397100448608, -1.3437747955322266, 1.5491913557052612, -0.9798378944396973, -0.29275137186050415, -0.33583974838256836, -0.6566533446311951, 2.942763090133667, 0.7935905456542969, 0.06699681282043457, -0.31036490201950073, 0.7058162093162537, 1.278495192527771, -1.113878607749939, -1.9509048461914062, 0.5059415102005005, 0.1865805834531784, 1.266142725944519, -0.8423236012458801, 0.06835490465164185, 0.48560890555381775, 0.7117379307746887, 1.0993558168411255, 1.2591816186904907, 0.1810530722141266, -1.3969671726226807, 0.9399403929710388, -0.9418792724609375, -0.18259049952030182, -0.4826187789440155, -0.16277959942817688, 0.9136519432067871, 0.48949649930000305, 2.1098341941833496, -1.1292414665222168, 0.10557045042514801, 1.0158582925796509, 1.7591801881790161, 1.4339383840560913, -0.8005591034889221, -0.01336826104670763, 0.2151438295841217, 0.24876807630062103, 0.5780196189880371, -0.06250112503767014, 0.6325625777244568, -1.2645155191421509, 0.6872126460075378, 0.48171260952949524, 1.3878746032714844, -0.2554491460323334, 0.8986391425132751, -1.2248554229736328, -0.7796074151992798, -1.2847518920898438, 0.8019363284111023, 1.2449123859405518, -0.8617861866950989, 1.1434470415115356, 0.7872248291969299, -0.7040520310401917, -0.430484801530838, 0.6518352031707764, 0.9883325695991516, -0.25969308614730835, -1.487797737121582, -0.9258655905723572, 1.713908076286316, 0.5631378889083862, 0.8470948934555054, -1.3512928485870361, 0.21032822132110596, -0.33964335918426514, -0.04770928621292114, -0.679656445980072, 0.9614758491516113, -0.747051477432251, 0.3778444528579712, 0.2634255588054657, -2.000469923019409, -0.06006985902786255, 0.18997681140899658, -0.7520249485969543, -0.6400770545005798, 1.0105408430099487, -1.1850674152374268, 0.24302789568901062, 1.3390910625457764, 1.2650772333145142, -0.19174467027187347, 0.22361135482788086, -1.0449928045272827, -0.4800506830215454, -0.047135721892118454, -1.0655380487442017, -1.727758526802063, 1.0590612888336182, -0.4016771912574768, 0.8519551157951355, -0.9430355429649353, -1.833726406097412, -0.6157032251358032, -0.9510533213615417, 2.6892542839050293, 0.12484176456928253, -0.16735345125198364, -0.9578527808189392, -0.3515372574329376, 1.9554035663604736, -0.5517948269844055, 0.7438009977340698, 0.9548996686935425, 0.7167257070541382, -1.2022205591201782, -1.1202815771102905, 0.3161669373512268, 0.4171692132949829, -0.14264880120754242, 0.6995577216148376, -1.1573184728622437, 0.3674318790435791, 0.6249169111251831, 0.1671179234981537, 2.209015369415283, -0.15021198987960815, -1.1948055028915405, -0.530990719795227, -0.17074449360370636, 1.168108582496643, -2.860063314437866, -0.7333317399024963, -0.9135063886642456, 0.611498236656189, 0.7546154260635376, -1.998998999595642, 2.1189844608306885, -1.0361436605453491, 1.1934700012207031, -1.5212560892105103, -1.039730191230774, -1.61973237991333, -1.223685622215271, -0.7520631551742554, -1.1054030656814575, 0.20916569232940674, -0.2826729118824005, 1.6425434350967407, 0.9036452174186707, 0.022977514192461967, 1.1895190477371216, -0.22340382635593414, 0.8946666717529297, -0.14451098442077637, -1.5404804944992065, 1.998094916343689, 0.11995397508144379, 0.9020849466323853, 0.8176082372665405, 1.8427550792694092, 1.1445597410202026, 0.6525877118110657, 0.5268722176551819, 2.5504937171936035, -0.8022820949554443, -0.6281859874725342, 1.1060020923614502, -1.138676404953003, 0.697055459022522, 0.3395787179470062, -0.008655905723571777, -0.8514415621757507, 1.071286678314209, -0.9674457907676697, 0.126140758395195, -1.878005862236023, 0.2181001752614975, 0.49165967106819153, -0.9280261397361755, 1.1726617813110352, -0.019184526056051254, -0.2178974747657776, -0.5634791851043701, 0.358476459980011, 0.5120893120765686, 0.23965106904506683, -0.012369845062494278, -0.5937826037406921, 1.7770881652832031, -0.49010512232780457, 0.2309533804655075, -0.41517558693885803, 0.42291662096977234, 0.14509239792823792, 1.1737793684005737, 2.703401803970337, 0.5972083210945129, 1.7771811485290527, 0.5211238265037537, 0.793951153755188, -0.25154197216033936, 0.724747359752655, 1.6578969955444336, -0.6120925545692444, -1.109695315361023, 0.26044005155563354, -1.0369884967803955, -1.2692756652832031, -0.6346227526664734, -0.07030099630355835, 0.9822180271148682, -0.04894978180527687, 3.1840803623199463, -2.476802110671997, 1.2015191316604614, -0.5954978466033936, -0.4411291480064392, -0.052430711686611176, 0.7284561991691589, 1.02750563621521, 1.1549204587936401, -1.2046253681182861, 0.137175515294075, -0.5261907577514648, 0.5406666994094849, -0.6475985050201416, -0.16544653475284576, -1.1007519960403442, -0.3559548258781433, -2.6162891387939453, 0.08730072528123856, -0.7089551687240601, -0.4746864438056946, 0.012660653330385685, 0.9594156742095947, -0.6265689730644226, -0.1624893695116043, -1.6614474058151245, 0.546355128288269, 2.168975591659546, 0.4025637209415436, 0.6939412951469421, -1.3825792074203491, 1.1497371196746826, -0.4348706901073456, 0.6599711775779724, -1.0051915645599365, 0.16783827543258667, -1.5220158100128174, 1.2431398630142212, 1.596028208732605, -1.643125057220459, 0.35170337557792664, 0.6916216611862183, 0.10349420458078384, 0.970949649810791, 0.6496815085411072, 1.4270708560943604, 1.4633954763412476, -0.14734771847724915, -0.13446886837482452, -2.2580602169036865, 2.1996328830718994, -0.31137803196907043, -1.7938854694366455, 0.70135897397995, -1.3518824577331543, -0.14852721989154816, 0.3834921419620514, 1.0225845575332642, -1.2017855644226074, 0.3024066686630249, 0.5093684196472168, 0.20300407707691193, 0.045386191457509995, 0.33697444200515747, 1.2391316890716553, 1.3462176322937012, -0.8759270906448364, -2.1030404567718506, -0.9510733485221863, -0.07287869602441788, 1.2486495971679688, 0.10105551034212112, -0.4261629283428192, -0.8785760998725891, -0.8741472363471985, -0.4482231140136719, -1.2182776927947998, -0.05653659626841545, 1.294394850730896, -1.3439702987670898, 1.2777153253555298, -0.8676862120628357, -0.7841324806213379, 0.00643738592043519, -0.23128573596477509, -0.14974594116210938, -0.45777544379234314, 0.8951883912086487, 1.7406060695648193, 1.9510588645935059, -0.31282809376716614, 0.9877465963363647, -0.7552945613861084, -0.46838387846946716, 0.07444297522306442, -1.3526115417480469, -2.2562644481658936, -1.2913358211517334, -1.0617767572402954, -1.1245702505111694, 0.9563246965408325, -0.6038693189620972, 0.25188812613487244, 1.1280618906021118, 2.3092141151428223, 0.36698174476623535, 0.32699793577194214, 0.5783300399780273, 0.1866064965724945, 0.7491082549095154, 0.07193712890148163, -0.8890119791030884, 0.28329241275787354, -0.010335137136280537, -0.8061611652374268, 1.0998667478561401, 0.4394989609718323, -0.07942374050617218, 0.37120646238327026, -0.4613477289676666, 0.28083211183547974, 0.09007677435874939, -0.5027782917022705, 0.4072532653808594, -0.08012042939662933, 0.15689504146575928, 0.34564799070358276, 0.5102663636207581, 1.8497010469436646, -0.30376508831977844, 0.22101478278636932, -0.5495860576629639, 0.6513087153434753, 0.40132880210876465, -0.04021978750824928, -0.1776515692472458, -0.502794623374939, -0.45615267753601074, -0.3060944378376007, 0.9105392694473267, 0.5621814727783203, -1.116426944732666, -0.15710735321044922, -0.13299517333507538, 2.684664726257324, -0.07536740601062775, 0.41382497549057007, 0.43809884786605835, -1.1658724546432495, -0.8380786180496216, 0.09442751854658127, 0.9333855509757996, -2.03456974029541, 0.3191849887371063, -0.932647705078125, 1.7981840372085571, 0.35046401619911194, -0.18648464977741241, -0.11094410717487335, -3.28969669342041, 0.6030306816101074, 2.6288390159606934, -1.4276148080825806, -1.4373786449432373, -0.49395444989204407, -0.6293861865997314, -0.7990380525588989, -1.1266727447509766, -0.413412481546402, -1.1422525644302368, -0.18428316712379456, -0.9324438571929932, 0.7206059694290161, -1.4408714771270752, -0.8231515288352966, -0.4260743260383606, 0.8235589861869812, -0.09929227828979492, 0.3417026698589325, 1.2307910919189453, -0.20449315011501312, -0.9153671860694885, 0.33100655674934387, 0.37057092785835266, -0.44658979773521423, 1.466435194015503, 0.4107137620449066, 1.096592664718628, -1.868283987045288, 1.3649615049362183, 1.155033826828003, 0.6433729529380798, -0.5804772973060608, -0.2932736575603485, 0.5106749534606934, -1.821219563484192, -0.9534258246421814, -1.0049864053726196, 0.015218150801956654, -0.41556307673454285, -0.9684237241744995, -1.42510986328125, 0.12437974661588669, -0.4264809489250183, 0.8059289455413818, 1.8998490571975708, -1.0131479501724243, 0.8354188799858093, 0.5129954814910889, 0.008629973977804184, -0.49413633346557617, -0.24467453360557556, -1.1605005264282227, -0.7789063453674316, -0.38732996582984924, 0.3725969195365906, -0.408368855714798, -1.0085716247558594, 1.7827011346817017, -0.11572308093309402, -1.2543919086456299, 0.435790479183197, 0.8210681080818176, 1.134047269821167, -0.7780627012252808, -0.7722693085670471, 1.0576846599578857, 1.5342164039611816, -0.752794623374939, 1.3633723258972168, -1.689563512802124, 1.487168550491333, 0.6493756771087646, -0.0028731906786561012, -1.3184280395507812, 0.2736174166202545, -0.8163478970527649, 0.6642598509788513, 0.005943933967500925, 0.909589946269989, 0.8050878643989563, 1.1554473638534546, -0.4241406321525574, -1.7055354118347168, 1.1758317947387695, 0.04261672869324684, 0.4649079144001007, 1.3623394966125488, 0.5564010739326477, -0.021421104669570923, -0.751926064491272, -0.10750623047351837, -1.103770136833191, 0.12436945736408234, -1.2294036149978638, -1.333741545677185, 0.34330081939697266, 0.32740676403045654, 0.5254063606262207, -0.4714849591255188, 0.6437845230102539, 1.030042052268982, -0.13164587318897247, 0.21772082149982452, 0.5273675918579102, 0.6220248937606812, -0.5840509533882141, -1.8383244276046753, -1.0608186721801758, 1.0262129306793213, 0.1407262235879898, 1.6800665855407715, -0.3802473247051239, 0.20035141706466675, 0.28340113162994385, 0.5151790380477905, 0.6154727935791016, -0.015576642006635666, -0.017416389659047127, 0.025301123037934303, -1.2962360382080078, 0.8863245844841003, 0.40014132857322693, 1.245370864868164, -1.5942398309707642, 0.2377004623413086, 1.0856847763061523, 2.2705793380737305, 1.6284714937210083, -0.3933115303516388, -0.9342904090881348, 0.706981360912323, 0.8036321997642517, 0.07679105550050735, 2.3927690982818604, 0.31836622953414917, -0.8488376140594482, -0.30990996956825256, 0.23171430826187134, -0.11685196310281754, 0.08471840620040894, -0.4339205026626587, -0.46781814098358154, 0.48499396443367004, 0.14785875380039215, 1.584449291229248, -0.7058544754981995, 0.2529037296772003, -0.32327499985694885, -1.3916268348693848, 1.145861029624939, -0.5004436373710632, -0.6823735237121582, -0.8941109776496887, 0.5931015610694885, 1.9152140617370605, -0.015589941293001175, -0.956340491771698, -0.44356071949005127, -0.5355875492095947, 0.29755067825317383, 1.0593618154525757, 1.6738208532333374, -0.012264556251466274, -1.023990511894226, 1.0534474849700928, -0.5462159514427185, -0.32143890857696533, -0.11457392573356628, 0.383504182100296, -1.005666732788086, -1.5230262279510498, -0.0218606386333704, -0.7221640944480896, 0.6670611500740051, 1.40701162815094, -0.9256536364555359, -0.9872434139251709, 0.3243407607078552, -0.6288321614265442, 0.12210583686828613, 1.0721274614334106, 0.7510967254638672, 1.7072818279266357, -1.2251965999603271, 0.016407648101449013, 0.5731435418128967, -0.42780131101608276, -0.10421846807003021, -0.9408882856369019, 0.24198608100414276, 0.14353980123996735, -0.9834906458854675, 0.03617991507053375, -0.3092438578605652, -0.9241689443588257, -1.0492136478424072, -0.3445466160774231, 0.6543189883232117, -0.14609073102474213, -0.9710582494735718, -0.15365208685398102, -0.03792310133576393, -2.0693976879119873, 0.5567822456359863, -1.6663074493408203, 1.048231840133667, 1.4242523908615112, 0.47323936223983765, 1.0746006965637207, -1.734177827835083, 2.303013563156128, 0.5464670658111572, 0.9506430625915527, 1.0185989141464233, 0.165536567568779, 0.0033954165410250425, -0.0020061596296727657, 0.1722429245710373, 0.05680675432085991, -0.9287747144699097, 0.4106040596961975, -0.288717657327652, 1.0655393600463867, -1.7190624475479126, 0.00922365766018629, 0.38240665197372437, -0.7496132254600525, -0.4771650433540344, -0.35960307717323303, -0.9070786237716675, -0.40612244606018066, 0.2896442413330078, 0.779485285282135, -1.1203653812408447, 1.9807804822921753, 0.012256658636033535, 0.017622454091906548, -1.9769277572631836, 0.938495397567749, 0.5268901586532593, 0.4715580344200134, 0.20536381006240845, -0.2753181755542755, -0.7899841666221619, 0.3999583423137665, 0.6993693113327026, -0.13265743851661682, -0.09733417630195618, 0.2892422676086426, -0.4149346649646759, -1.4088712930679321, 1.2547847032546997, 1.5237964391708374, -0.3148292601108551, -0.5763065218925476, 1.304805040359497, 0.19215720891952515, -0.5822939872741699, -1.169237494468689, -1.873988389968872, -0.34584981203079224, -1.601846694946289, 0.1692768931388855, -2.828176975250244, -1.3955100774765015, -0.20084388554096222, 0.4887699484825134, 0.5315940976142883, -0.18152755498886108, 0.24005484580993652, -1.7973086833953857], [-1.346724510192871, -0.31360018253326416, 1.4693384170532227, 0.5165178179740906, -1.418183445930481, -0.836821436882019, 1.9976781606674194, 1.0177210569381714, -1.8537956476211548, 0.1459372192621231, 2.20204496383667, 1.3685985803604126, 1.046841025352478, -1.2809621095657349, 2.6692116260528564, 0.45131659507751465, -1.300634741783142, -0.782617449760437, -1.2033032178878784, 1.0992478132247925, 1.3700191974639893, 1.0466175079345703, 0.028018327429890633, 1.0285792350769043, -0.8241725564002991, -0.7428095936775208, -0.039658721536397934, 0.6528005599975586, 0.016394253820180893, 1.2457759380340576, -2.654041290283203, -0.7248850464820862, 1.1683717966079712, -1.763974905014038, -1.7830926179885864, 0.062487948685884476, 1.303674578666687, -0.406997948884964, 1.261568307876587, -0.07970523089170456, -0.138011172413826, -0.9297587275505066, -0.41656237840652466, 2.485663890838623, -0.4983339011669159, 1.5072340965270996, 0.5628217458724976, 0.14814390242099762, 0.6308234930038452, -0.23150914907455444, 0.14758694171905518, 0.5474140644073486, -1.7400054931640625, 0.41445377469062805, 0.6983621120452881, 1.6993011236190796, -0.3406676650047302, -0.11838916689157486, -0.3985605239868164, -0.7719639539718628, -0.6631149649620056, -1.0677841901779175, 0.06867317855358124, 0.5300850868225098, -0.3719247281551361, 0.4067137539386749, -0.7890623807907104, 0.747454822063446, 0.5425904989242554, 1.2011855840682983, 0.30086299777030945, 1.6487858295440674, 0.12219654768705368, 0.8219203352928162, 0.2856024503707886, 0.8320050835609436, -0.5498216152191162, 0.6156987547874451, -0.2928827702999115, -0.7998387217521667, -0.21201083064079285, 0.1729017198085785, -1.8902487754821777, -1.517066478729248, -0.7162883877754211, -0.3890223503112793, 0.37131860852241516, 1.3937172889709473, -0.78642737865448, 0.4267290532588959, 0.32357120513916016, -1.1477009057998657, -0.927484393119812, -0.059836745262145996, -0.28294482827186584, 2.1786305904388428, -0.8414029479026794, -0.91707843542099, -2.001720905303955, -0.08222749829292297, -0.28595975041389465, 0.425032377243042, 0.24881771206855774, 0.2071329951286316, -1.054870843887329, -1.9241737127304077, 0.6448751091957092, 0.37193408608436584, -0.6361674666404724, 0.9042282700538635, 1.0983761548995972, -0.8004074096679688, -1.4117262363433838, -1.064742922782898, -1.0984821319580078, -0.6846252679824829, 1.4861048460006714, -0.5898127555847168, 0.9234127402305603, -0.2321488857269287, -0.2781890034675598, 0.9618785977363586, -0.547302782535553, -0.575404703617096, -2.1207473278045654, -0.9217184782028198, -1.6560808420181274, -0.7956413626670837, -2.530364990234375, 0.5380038619041443, 0.7477881908416748, 0.6565769910812378, 0.1431621015071869, 0.03592503443360329, -0.5493888854980469, -0.4163931608200073, -0.11462368816137314, -0.9003102779388428, 1.1138899326324463, -0.7027490735054016, -0.17977695167064667, 0.9240384101867676, 1.2129578590393066, -1.2841373682022095, -0.2695457339286804, 0.2786610722541809, 1.0988941192626953, 0.8956362009048462, 1.0598665475845337, -0.32543253898620605, -0.38817667961120605, -1.3716630935668945, -1.1499603986740112, -0.938105583190918, 0.2727277874946594, 0.10942171514034271, 2.251786470413208, 1.0105571746826172, -2.1674461364746094, 1.5127092599868774, -0.3085653781890869, -1.6018126010894775, -2.0488154888153076, 1.7795159816741943, -0.5768001675605774, 0.1636718064546585, -0.8088246583938599, 0.14017152786254883, 0.4243967533111572, 1.0838062763214111, 0.023773200809955597, -1.3755903244018555, -0.7607840299606323, 0.6358648538589478, 0.15388262271881104, -0.7370449304580688, -0.6709092855453491, 0.08386129140853882, 0.686548113822937, 1.144931435585022, 0.9730960726737976, -1.2845110893249512, 0.6794273853302002, 0.3617376387119293, 1.252827525138855, -0.6138299703598022, -0.9430577754974365, -0.09491284936666489, -2.1318845748901367, 1.693479299545288, -1.8343641757965088, 0.704323947429657, -0.5430057048797607, -1.0195810794830322, -0.2210581749677658, -1.9216859340667725, -0.08885999768972397, -0.4913116991519928, 0.3419116139411926, 0.13474681973457336, 0.6676187515258789, -1.2594304084777832, -0.5154833197593689, -0.98459392786026, 0.5981988310813904, -0.2831210196018219, -0.8820183873176575, -0.39106428623199463, -0.44782787561416626, 0.19806219637393951, 0.5132942199707031, 0.7835352420806885, -0.7263669371604919, 1.0801379680633545, 0.9682062268257141, 0.3624299168586731, -0.6780647039413452, -0.8859275579452515, 0.31633639335632324, -0.3446149230003357, -1.3119491338729858, -1.2273105382919312, 0.6162513494491577, 0.7907884120941162, -1.5013964176177979, 0.6124882102012634, -0.29575642943382263, -1.2334482669830322, 0.6239196062088013, -1.306578516960144, 0.29336556792259216, -1.2459697723388672, -1.086629033088684, 1.1116458177566528, 1.3587883710861206, 0.7597776055335999, -1.19875168800354, 0.46668851375579834, 1.4218196868896484, 0.31492879986763, 1.6534552574157715, -0.3372970223426819, -1.3623837232589722, 1.6633315086364746, -1.4010320901870728, -1.0294543504714966, -1.1912763118743896, -1.2649685144424438, -0.24247419834136963, -0.4275059103965759, -0.09499571472406387, 1.5843088626861572, 1.3494161367416382, 0.255012571811676, -0.6930437088012695, -0.14040826261043549, 0.5674044489860535, -0.2853362262248993, -0.9190983772277832, 1.093645691871643, -0.17160916328430176, -1.6041630506515503, 0.7594521641731262, 0.7851747870445251, -0.7268580198287964, -1.806822419166565, -1.1950167417526245, 0.6307910084724426, -0.5320473909378052, 0.7354198098182678, -0.4102875292301178, 0.7470447421073914, 0.5570672154426575, 0.20483650267124176, 0.21536746621131897, -0.649113655090332, -0.4367508292198181, -1.775997281074524, -0.5761761665344238, 0.15310530364513397, -0.17819955945014954, 0.16142132878303528, -1.2626845836639404, -0.2044437825679779, -1.5451023578643799, 0.7729384303092957, 0.954984188079834, 0.5161058306694031, 0.6385483741760254, 2.359631299972534, 0.10415153950452805, -0.5944822430610657, 0.1535656601190567, -0.8077238202095032, -0.9302770495414734, -1.2909291982650757, -1.3485363721847534, -1.0178141593933105, 0.008755476213991642, 1.013988971710205, -0.5658409595489502, 1.7782783508300781, -0.12551897764205933, -0.726757287979126, -1.2555979490280151, -3.0171492099761963, 1.3326135873794556, -0.7463831305503845, -0.3803437352180481, 0.020166609436273575, -0.09295381605625153, -1.6415973901748657, 1.380905270576477, -0.26511117815971375, -0.9588102102279663, 1.923878788948059, -0.7709699869155884, 1.2514855861663818, 1.1673951148986816, -0.08991022408008575, -0.8090645670890808, -1.5482866764068604, -1.1375837326049805, -0.5464139580726624, 0.5490080714225769, -0.43240663409233093, -1.707503318786621, 0.9239746332168579, -0.9517574906349182, -1.1423749923706055, -0.1871454417705536, 1.7159948348999023, 1.5125495195388794, -0.09613256901502609, -2.2086570262908936, -0.043672122061252594, -2.3357841968536377, 0.169870987534523, -1.480338454246521, -1.021972894668579, -0.6269313097000122, -0.12837596237659454, 0.9246991872787476, -0.22619284689426422, 0.5749340057373047, 0.41102302074432373, 0.09478195756673813, -0.9161481261253357, 1.8988004922866821, -1.9558390378952026, 0.08104969561100006, 1.0610816478729248, 1.0126442909240723, -0.358871728181839, 1.1982313394546509, -0.4590905010700226, -0.953285276889801, 0.7998101115226746, -0.8017129302024841, 2.0190811157226562, 0.7816330790519714, 1.2782142162322998, -0.64010089635849, 2.011383533477783, 1.555903434753418, -0.1958208531141281, 0.5149935483932495, -0.247049480676651, 0.8598707318305969, -1.2687883377075195, 0.2242645025253296, -0.6280114054679871, -1.0226922035217285, 0.424577534198761, 0.8617458343505859, -0.9783198833465576, 0.2926636338233948, 0.013074306771159172, -2.6257503032684326, -1.7135382890701294, 0.7097461819648743, 0.9170796275138855, -1.4344450235366821, -0.683196485042572, 1.3437840938568115, 0.6814252734184265, -0.23654212057590485, 0.252204030752182, 0.06487195938825607, 0.4842239022254944, -0.5523821711540222, -1.0388712882995605, -0.023776115849614143, 0.38102880120277405, -0.12128351628780365, -0.2854647934436798, -0.3330053687095642, -0.07933308929204941, 1.8824459314346313, 0.14537325501441956, -1.7132090330123901, 0.30223819613456726, 0.8705890774726868, 0.4838227331638336, 0.0007161417743191123, 0.15815585851669312, 1.4521225690841675, 0.10397221893072128, -0.6594114899635315, 1.7522189617156982, -1.5158796310424805, -0.9388923048973083, -0.10088388621807098, 1.2950094938278198, -1.8183999061584473, 1.4775941371917725, 3.784423828125, 0.4080846905708313, -1.5204194784164429, -0.6402928829193115, -0.2869546711444855, 0.04021194577217102, -0.5078534483909607, -0.22955504059791565, 0.9904776811599731, 0.6238440871238708, 0.5562264919281006, -0.25346487760543823, -0.8967390060424805, 1.0485576391220093, 1.3990111351013184, 3.146817922592163, 3.155371904373169, 0.1138748899102211, 1.2930420637130737, 1.8651607036590576, -0.42421871423721313, 0.014418235048651695, -0.6484770774841309, 0.5456961989402771, -0.7058406472206116, 0.5098683834075928, -1.2086431980133057, -0.8263859152793884, 0.9560775756835938, -0.2847498953342438, -0.44758307933807373, 0.6901007294654846, 1.0747950077056885, 1.151534080505371, 0.5559852719306946, 0.27242952585220337, 0.744234561920166, -1.0036336183547974, 0.5276027321815491, -2.055972099304199, -0.4451253116130829, 0.6895419955253601, 1.0385156869888306, -0.06495747715234756, 2.075655460357666, -0.6202865242958069, -0.6573958992958069, 0.9546463489532471, -0.5135123133659363, -0.08942200988531113, 0.21045617759227753, 0.751889705657959, 1.7732124328613281, 0.6800684928894043, 1.0754148960113525, -1.4717683792114258, -0.7259752154350281, 1.4157600402832031, 0.11970196664333344, -1.5671703815460205, 1.0832724571228027, -0.5655072927474976, -1.003125786781311, 0.3895929753780365, -1.6788783073425293, 1.1267915964126587, 0.34139004349708557, 0.9931796789169312, 0.8018218874931335, -0.7809562087059021, 0.05447133630514145, -0.9726601839065552, 0.1609237790107727, -2.0586626529693604, -0.8089068531990051, -0.6958034038543701, 1.0059576034545898, 1.600704550743103, -1.2101584672927856, 1.1202269792556763, 1.2576088905334473, -1.4585567712783813, 0.5238671898841858, 0.0379977710545063, 0.08007268607616425, 0.8183659911155701, 0.6908366084098816, 0.11117095500230789, -0.027848239988088608, 0.213194340467453, -1.2259113788604736, -0.010069767013192177, 1.0324182510375977, 1.147864580154419, -0.8834497332572937, -0.7601786255836487, 1.8850916624069214, -1.1323405504226685, -0.3411550223827362, 1.958122968673706, -0.4277271628379822, 0.8342192769050598, -0.9050688147544861, -0.2916881740093231, -1.046058177947998, 0.39751729369163513, 0.8560250401496887, 0.5912021994590759, -0.6184574365615845, 1.243404507637024, -0.21702516078948975, 0.24641740322113037, -0.4335939884185791, -0.07409946620464325, 0.8274844288825989, -0.2001701444387436, 0.9590626955032349, -1.1444593667984009, -0.4814979135990143, 1.563604474067688, -0.2020573914051056, -1.3246923685073853, -1.531439185142517, 0.08917321264743805, -0.9925189018249512, 1.6814409494400024, 0.25116729736328125, -2.120850086212158, -0.7605470418930054, -0.5522786974906921, 0.3420541286468506, -0.9316344261169434, -1.6321709156036377, -1.4469356536865234, 0.7753540873527527, -0.810088038444519, -0.5153021812438965, 0.33077189326286316, -1.6425966024398804, -0.5390177965164185, 0.20446325838565826, -1.127038598060608, -0.794929027557373, 1.8807328939437866, -2.391085624694824, 0.3719296455383301, 0.9043765664100647, -0.271560400724411, -0.018820876255631447, -1.215771198272705, -0.5237921476364136, -0.038115642964839935, 1.010328769683838, 0.48425254225730896, -0.780777633190155, -0.8980019688606262, 0.574476420879364, 0.3703228533267975, 0.3762948215007782, 0.03725453093647957, 1.5075998306274414, 0.41667598485946655, 0.9178416728973389, 1.0334304571151733, 0.0388495959341526, -0.5127196311950684, -1.706966757774353, 1.0533533096313477, -0.05600231885910034, -0.16167710721492767, 0.31196901202201843, -0.44810351729393005, 0.4215070307254791, -1.8149019479751587, -0.41217052936553955, 0.2492307722568512, 0.19789791107177734, -0.8448396325111389, -0.9089016914367676, -0.4661901891231537, -0.9374567866325378, -0.3205341100692749, -0.7414934039115906, 0.012987258844077587, 0.3049076199531555, -0.741189181804657, -0.35825982689857483, -0.47916293144226074, -1.2656348943710327, 0.7808234691619873, 0.34384042024612427, -0.5559080243110657, -1.8502775430679321, 0.8392471075057983, 1.0623631477355957, 0.5043658018112183, 0.3075021207332611, 0.6643924713134766, 0.5511800050735474, -0.7539029717445374, -0.8205122351646423, 0.055653076618909836, -0.8053945302963257, -2.228909969329834, -0.9219543933868408, -0.6794527173042297, 0.698776125907898, -0.072491854429245, -0.5921292304992676, -1.5395265817642212, -1.760678768157959, 0.5805977582931519, -0.06155413016676903, 0.12040118873119354, -0.21866339445114136, -0.2349102795124054, 0.4212305247783661, 1.1356269121170044, -1.3369964361190796, -0.3148980140686035, -1.159143328666687, -1.0515546798706055, -0.6497219800949097, -0.09784151613712311, -1.2039138078689575, -0.07955331355333328, -1.675223708152771, 0.8671069145202637, 0.4553470015525818, -0.9927228093147278, 0.44944316148757935, -1.34601628780365, -0.9234347343444824, -0.5375963449478149, 1.0909948348999023, -0.609423041343689, 0.3437735438346863, -0.43835902214050293, 0.012352483347058296, -0.18970654904842377, 1.5221706628799438, -1.2717994451522827, 1.3883503675460815, -1.0585237741470337, -0.17486156523227692, 0.6489429473876953, 0.9698625206947327, 0.34081128239631653, -0.0910097062587738, 0.9799512028694153, -1.1018812656402588, 2.3779191970825195, 1.224086046218872, 1.6266117095947266, 1.2341252565383911, 1.1153260469436646, 1.8783113956451416, -2.0327610969543457, -0.45161399245262146, 0.0881701409816742, 1.9651912450790405, -1.2373522520065308, 0.6467448472976685, -1.9639866352081299, -2.5503950119018555, 0.08155269920825958, 0.07674086093902588, -0.9691136479377747, 1.461169958114624, 0.5950556397438049, -0.3916996121406555, -1.0163533687591553, -0.9997793436050415, 0.5502942800521851, 0.59378582239151, -1.6918416023254395, 1.514541506767273, 1.6360594034194946, 0.6583579778671265, -1.2090672254562378, -2.0810766220092773, 1.0715312957763672, 0.25380608439445496, 0.81494140625, -0.29974716901779175, 1.321183443069458, 0.08399418741464615, -1.0401852130889893, 0.5557079315185547, -0.7169650197029114, -0.6170528531074524, -0.29020455479621887, -1.5612115859985352, -1.4176485538482666, -0.7492865324020386, 0.13973736763000488, 0.20182771980762482, -0.866300642490387, -0.3814234137535095, 0.9524716734886169, 1.4809284210205078, 0.3085457682609558, 0.2921341359615326, 0.6346650123596191, 0.4260905981063843, 0.7976489663124084, -0.44002705812454224, -0.3934689462184906, -0.5945088267326355, 0.001547386054880917, 0.674168050289154, 1.1065565347671509, 0.5833227038383484, 1.4192415475845337, -0.3549558222293854, -0.37634095549583435, 0.36657479405403137, -2.8120360374450684, -0.36679208278656006, 0.5367695689201355, -2.097876787185669, 0.579662024974823, 0.49211645126342773, 0.4444112181663513, -1.0689691305160522, -0.8550111055374146, 1.6269947290420532, -0.8480037450790405, 0.33650073409080505, -0.544119656085968, -0.8182403445243835, 1.336387276649475, 0.7638569474220276, 0.41945359110832214, -0.7322717308998108, -1.5045151710510254, 1.4929708242416382, -0.15361414849758148, -1.6224242448806763, 1.5627501010894775, -0.33882936835289, -0.08214862644672394, 0.9774631261825562, 1.2638530731201172, 0.52939373254776, -0.8837980628013611, 0.37298136949539185, 1.689003348350525, 0.8694700002670288, 0.7819084525108337, -0.04337237775325775, 1.3254952430725098, -1.587746024131775, 0.00555560365319252, -0.2144874483346939, -0.4088447690010071, -0.5958530902862549, 0.1874355524778366, 0.3192785680294037, -0.2844107151031494, 0.36865541338920593, -0.15810810029506683, 0.1075434759259224, -0.5551159977912903, 0.32580605149269104, 2.297461748123169, -1.7561463117599487, -0.10799200087785721, -1.0571131706237793, -0.5239443778991699, 1.8962757587432861, -1.5853581428527832, 0.36841511726379395, -1.453191876411438, 1.5522743463516235, -0.8223480582237244, 1.5716582536697388, -0.05011193826794624, 1.4081782102584839, -0.9045115113258362, -0.2203090637922287, -1.6914118528366089, 0.8109425902366638, 0.6815792918205261, -0.7724379897117615, 0.8217087388038635, 0.03626817837357521, 0.7966132760047913, -1.18904447555542, 0.16214053332805634, 0.2673346996307373, 0.8487527370452881, 1.1080012321472168, -1.1802940368652344, 0.28823739290237427, -1.1404446363449097, -0.12010249495506287, 1.5963131189346313, 1.0699131488800049, -0.44215112924575806, -0.7906129956245422, -0.4764899015426636, 1.6743794679641724, -0.431398868560791, -0.5709060430526733, 0.018396632745862007, 0.03400402143597603, -0.34006017446517944, 0.4084581434726715, -1.7885093688964844, -0.9251112341880798, 0.9938153028488159, 0.20908306539058685, -0.6037607789039612, 1.139135718345642, -0.4631351828575134, 0.23534610867500305, 1.5367494821548462, 1.1115409135818481, -0.30938708782196045, 0.5009493827819824, -1.4324133396148682, 0.3982153534889221, 1.2862056493759155, -0.5063899159431458, -0.10078989714384079, 1.5020753145217896, 0.03555244952440262, 1.3934823274612427, 0.14342357218265533, -0.6896281242370605, 0.47132033109664917, -0.11511560529470444, -2.112388849258423, 1.2386019229888916, -0.47281748056411743, -1.1537638902664185, 0.6217437386512756, -0.1818416565656662, -0.4344024658203125, -0.673677384853363, 1.3188238143920898, 1.0354840755462646, -0.8680158257484436, -0.8880355954170227, 0.517742931842804, -0.3396052122116089, -0.26856333017349243, -1.1771175861358643, 0.4758436977863312, -0.20543155074119568, 0.1850355863571167, -0.3180554211139679, 0.36276862025260925, -0.15191902220249176, -0.6985804438591003, 0.01613190397620201, 0.7952108979225159, -0.541197657585144, -0.6999416351318359, -1.1561731100082397, 0.46208494901657104, 0.8396968841552734, 1.924539566040039, -0.3118656873703003, 0.5113791823387146, -0.35275766253471375, -1.161555290222168, 0.357158899307251, 1.8058193922042847, 1.2657794952392578, 0.9629530906677246, -0.4281476140022278, 0.3968282639980316, -0.35205623507499695, 1.0849266052246094, 1.4944485425949097, -1.9147394895553589, 1.404463291168213, 0.5490476489067078, 0.34375089406967163, -1.0316780805587769, 0.23914670944213867, -2.1054205894470215, 1.3785830736160278, -0.18102143704891205, -0.390963077545166, -2.012709856033325, -0.44752004742622375, 1.6633530855178833, -0.43431833386421204, -0.4948631823062897, -1.9433331489562988, 1.0459659099578857, 0.28856655955314636, 1.3705129623413086, 0.7423585653305054, -0.9403917789459229, -0.6390606164932251, -0.9186442494392395, -0.48248958587646484, -0.9684853553771973, 2.0154216289520264, 1.1616722345352173, 1.9853806495666504, -0.016752595081925392, -0.2458404004573822, -0.639691174030304, -0.6468966603279114, -0.5163770914077759, 0.7020109295845032, -2.181135654449463, -0.4443801939487457, -0.21374046802520752, -1.0519330501556396, 0.9747949838638306, -0.030223475769162178, -0.776907205581665, 1.1599000692367554, -0.32403185963630676, 0.5169863104820251, 1.2036776542663574, 0.8866873383522034, 0.8969714045524597, -0.5400722622871399, -1.913914442062378, 0.9192309975624084, 2.267062187194824, 0.1033589243888855, 0.7674977779388428, -0.10322147607803345, 2.7566771507263184, -0.39501503109931946, 2.569959878921509, -0.961815357208252, 0.9357162714004517, -0.37663742899894714, 0.14209720492362976, 0.7926563024520874, 1.1224926710128784, 0.6025708913803101, 0.5872415900230408, -1.331599235534668, -0.029121017083525658, -0.7740082144737244, 1.9874422550201416, 1.8864508867263794, -0.15994995832443237, -1.432689905166626, -0.12765948474407196, -0.7974514365196228, 0.8638880848884583, 0.31046348810195923, -0.9528139233589172, -1.655178427696228, -0.03678639978170395, 0.731486439704895, 0.31696614623069763, 0.9334663152694702, 1.4199823141098022, 0.9891311526298523, 0.633284866809845, 0.9914470911026001, -1.183107614517212, -0.561301052570343, -0.4374834895133972, 1.7040303945541382, 0.20728592574596405, -0.2714051306247711, -0.8655390739440918, 1.4416512250900269, 1.5142767429351807, 0.1775403767824173, 1.1455868482589722, 0.09884849935770035, 0.7925739884376526, 0.8819961547851562, 0.21367886662483215, -0.6416612863540649, 0.5870313048362732, -0.24687045812606812, 0.9201185703277588, 0.9200584888458252, -0.2691308259963989, -2.9937894344329834, -0.5363680124282837, -1.1056897640228271, 1.6592695713043213, 0.031683605164289474, -1.2514444589614868, -1.573065161705017, -0.22360628843307495, 0.4953555166721344, -0.925170361995697, -0.9891675114631653, 0.12977935373783112, 1.5204074382781982, 0.5520532727241516, 1.7879605293273926, -0.34268462657928467, 0.4274628460407257, -0.6933297514915466, 0.310934454202652, 1.2371288537979126, 0.33206403255462646, 1.2963922023773193, -0.36908960342407227, 0.5199875235557556], [-0.41269150376319885, -0.3921740651130676, 1.2088879346847534, -0.4296107888221741, 0.018800882622599602, -0.42274150252342224, -0.533765435218811, -0.4605514109134674, 1.585284948348999, 0.49193787574768066, -1.294441819190979, 0.6224645376205444, -0.36654964089393616, -0.9583368897438049, 0.05437833070755005, 0.9685776829719543, 0.17101560533046722, 0.0986528992652893, -1.1030960083007812, 0.632563054561615, -0.5767386555671692, -2.851402997970581, 0.553798496723175, 0.15085062384605408, -0.38359978795051575, 0.5250654816627502, 1.7484140396118164, 0.28239768743515015, -1.713077187538147, -0.16637210547924042, 0.012938987463712692, 1.7696505784988403, 0.5718412399291992, -1.76627779006958, 0.3938799202442169, 1.406805396080017, -1.730817437171936, -0.46649280190467834, -0.2250840961933136, -0.6795854568481445, -3.187775135040283, -1.9595942497253418, 0.41965624690055847, 0.664608895778656, -1.0550391674041748, -1.2227749824523926, 0.8057486414909363, 0.024057794362306595, 1.2558001279830933, -2.3431601524353027, 0.2643415033817291, 0.33121955394744873, -1.4845856428146362, -1.2149368524551392, -0.4244343638420105, -0.6962716579437256, -0.46815335750579834, -0.9198560118675232, 0.7876079678535461, 0.5034428834915161, 0.8503977060317993, -1.1128813028335571, 0.49981093406677246, -0.3811277449131012, -1.4015419483184814, -1.2643345594406128, -0.5011723637580872, -1.109520673751831, 1.907265305519104, 2.6418612003326416, -0.06096574291586876, -0.07289224863052368, -0.782433032989502, -0.6495944261550903, 0.38637688755989075, -0.6140539646148682, 1.1445703506469727, 1.5312520265579224, -1.1211557388305664, 1.5954657793045044, 0.858327329158783, -0.05109763145446777, 1.0076836347579956, 1.0202537775039673, 0.17946410179138184, -1.7929526567459106, 0.20359490811824799, -0.4350419044494629, 0.2075779139995575, 0.49150756001472473, 1.998389720916748, -1.4323879480361938, 0.5114455819129944, 0.31228962540626526, 0.6191171407699585, 0.5427492260932922, -1.2312496900558472, 0.7027484774589539, 1.4881128072738647, 1.5833868980407715, -0.13213609158992767, -0.1638227105140686, 0.5341720581054688, 0.07122315466403961, 0.3239912986755371, 0.4746027886867523, -1.0564565658569336, -0.3712191581726074, 1.9681050777435303, -1.882368564605713, 0.9161486625671387, 1.0181570053100586, 1.4634140729904175, -1.972939133644104, 1.6616591215133667, 1.0286630392074585, 1.5937689542770386, 0.48351389169692993, -1.383558750152588, 0.5594071745872498, 0.16841115057468414, 0.879143476486206, -0.26438137888908386, 0.5563512444496155, 0.08544284105300903, -1.210541009902954, -0.42355266213417053, 0.8862681984901428, -0.03165128827095032, -0.4970347285270691, 1.4505269527435303, 1.0168113708496094, -0.47291791439056396, 0.17439770698547363, -2.0212137699127197, 0.7836819887161255, -0.36987394094467163, 1.5502156019210815, -0.6054670214653015, -0.7311908006668091, -1.5476207733154297, -0.25724709033966064, 0.5216732025146484, -0.18649263679981232, -0.3161581754684448, 0.004579514730721712, 1.1162136793136597, 0.5246171951293945, -0.7346295118331909, 1.4631340503692627, 0.3584195077419281, 0.4671229422092438, 0.11847230046987534, -0.6652256846427917, 1.3198164701461792, 2.142436981201172, -1.1431134939193726, -1.813908338546753, 2.5504214763641357, 0.29965919256210327, -0.6135933995246887, 1.9088263511657715, 0.18067416548728943, 0.779455840587616, -0.5398953557014465, -0.43636763095855713, -0.08988562971353531, 1.965532898902893, -0.1940295696258545, 2.5606138706207275, -0.009560450911521912, -0.5256562232971191, -0.0012863011797890067, -1.1431957483291626, 0.4537197947502136, -0.47699642181396484, -1.4437835216522217, -0.20406511425971985, -0.6511748433113098, 0.26436713337898254, 0.3277316689491272, 0.39712756872177124, -0.7726582288742065, -0.26751670241355896, 0.521817684173584, -0.14920590817928314, 0.6185818910598755, 0.564451277256012, 1.4898254871368408, 0.753527820110321, -0.328814297914505, 0.7981811761856079, 0.7907899022102356, 0.8451893329620361, 0.4140583574771881, -0.9020270705223083, 0.3511638641357422, -0.8715265989303589, -0.2112666517496109, 0.610615611076355, -0.7304716110229492, -0.6116457581520081, -0.5543280243873596, 0.713905394077301, 1.5913269519805908, 0.17035169899463654, -0.18832845985889435, -1.2891643047332764, 0.9208048582077026, 0.8503280878067017, -0.8015028834342957, 0.3214946985244751, 1.5070631504058838, 0.4816722869873047, 0.5956337451934814, 0.7711260914802551, 0.9234247803688049, -1.0975581407546997, -1.7509225606918335, -0.16397635638713837, -1.2976100444793701, 0.33922329545021057, 0.9645422101020813, -1.8581838607788086, -0.44454824924468994, 0.04684995859861374, 0.7278138995170593, 2.069683790206909, 0.7451398968696594, -1.0116342306137085, 0.2947824001312256, -0.13515903055667877, 0.25586453080177307, 1.9054644107818604, 1.0762842893600464, 1.1020294427871704, -0.7024552226066589, -1.0799425840377808, 0.44341957569122314, -0.6653937101364136, 1.351749300956726, -0.588857889175415, 0.9709196090698242, 0.25945377349853516, 0.33269202709198, -1.7213692665100098, 1.7439295053482056, 2.1704351902008057, -0.45954641699790955, 0.3828427791595459, 0.9005374908447266, 1.455302357673645, 1.2019597291946411, 0.7695732116699219, 0.5809657573699951, 0.674363911151886, 0.41710007190704346, -0.5930773019790649, -0.10315851867198944, -0.5383564233779907, 0.5214830636978149, 0.7740283608436584, 0.7510191798210144, 1.205621600151062, -0.7033347487449646, -2.6984944343566895, 1.0129263401031494, 0.24913042783737183, 0.16793513298034668, -0.6218812465667725, 0.2901894152164459, -0.8638073205947876, -0.545633852481842, 0.6332890391349792, 1.87459135055542, 0.0538034662604332, 1.2447909116744995, 0.5621732473373413, 1.4230674505233765, 1.8477675914764404, -1.6679290533065796, 0.5670725107192993, 1.5554066896438599, -0.29633578658103943, -0.8161510825157166, 1.591416597366333, -1.5226683616638184, 0.7027443647384644, -0.10434875637292862, 0.42794784903526306, 0.9301792979240417, 1.6724944114685059, 0.47975218296051025, -1.0742778778076172, -0.45557522773742676, -0.8930031061172485, 0.4127500355243683, 0.9026862978935242, 0.4617232084274292, 0.48133397102355957, -0.3963552415370941, -1.4697468280792236, 0.4280507564544678, 0.723403811454773, -0.7958499789237976, -1.0134774446487427, -0.15587450563907623, 0.6508169174194336, 0.27120307087898254, -1.5774661302566528, -0.23070532083511353, 0.0802120715379715, -1.192049503326416, -0.9691748023033142, 0.4284743666648865, 0.41689202189445496, 0.620966911315918, -0.10031649470329285, -0.5600308179855347, -0.2288074791431427, 0.4629286825656891, 1.3391780853271484, 0.2763778865337372, -0.8821476697921753, 0.19356106221675873, -0.4538402557373047, 1.6702975034713745, 0.25419196486473083, -0.2494804710149765, 0.636885404586792, 0.3895270824432373, -0.347582072019577, 1.8228223323822021, -1.0787954330444336, 1.3559776544570923, -0.007532889023423195, 0.011047995649278164, -0.46255558729171753, 1.2371695041656494, -0.8827542066574097, 2.542116641998291, 0.6939338445663452, 0.41433659195899963, 0.522861897945404, 0.7852823734283447, 0.1298140287399292, 0.07483673840761185, -0.18606947362422943, -0.9525041580200195, -0.940664529800415, 0.9305288195610046, -0.3029112219810486, -1.211079478263855, -1.075656533241272, 1.3777271509170532, -0.9381946921348572, -1.000861406326294, -1.091896891593933, -0.8933296799659729, 1.5122309923171997, 1.9508025646209717, -0.6261098384857178, 1.6475480794906616, -0.20002548396587372, 0.8754597902297974, -0.9808512330055237, 1.4464308023452759, 1.4210935831069946, 0.18991222977638245, -0.7079803943634033, -1.403922200202942, -0.49191129207611084, -0.045851629227399826, -0.9947513341903687, -1.4176257848739624, -0.6250327825546265, 0.7199562788009644, 0.32230645418167114, 0.14452512562274933, 0.8841109871864319, -0.46279191970825195, -1.1885452270507812, -0.731806218624115, -2.519331455230713, -1.230470061302185, -0.8199544548988342, 0.5276353359222412, 0.6165562272071838, -1.831923484802246, -0.2895401418209076, -1.6516834497451782, 0.550852358341217, -1.3320890665054321, 1.3089040517807007, 0.9297890663146973, -1.1110225915908813, 2.3689353466033936, 1.650061011314392, 0.9424375295639038, 0.5380558967590332, -1.0201747417449951, 0.0848117396235466, -0.2117811143398285, 1.6954636573791504, -0.9587443470954895, 0.4295155107975006, -0.6559810638427734, 1.4294544458389282, 0.010877368971705437, -0.40280061960220337, 0.7937731146812439, -1.748706340789795, 0.019472764804959297, -1.0636905431747437, -0.5963165760040283, -0.3381716012954712, -0.3235718607902527, 0.09136128425598145, -1.775917649269104, -0.36044245958328247, -0.37802064418792725, -0.6843324303627014, -1.024229884147644, 0.30695223808288574, 0.0019145540427416563, 2.5835821628570557, 0.837867796421051, -1.7662444114685059, 0.160562202334404, 0.3524492681026459, 0.6386549472808838, 0.3672267496585846, -2.023730993270874, -0.965019702911377, -1.1827685832977295, -0.4122556149959564, 0.6991433501243591, 0.6925867199897766, -0.719547688961029, 0.6948263645172119, 0.33867478370666504, 0.6982267498970032, -0.15180377662181854, 1.6838856935501099, -0.044734660536050797, 0.0950414314866066, 0.15948915481567383, 0.09046553820371628, -1.0880470275878906, 0.4986191391944885, 0.020100697875022888, 1.2373994588851929, -1.3610031604766846, -2.0255556106567383, 2.687206506729126, 0.5047551989555359, 0.15198665857315063, 0.4762411117553711, -0.2477167844772339, 0.1679614782333374, 0.8167428374290466, 2.0279934406280518, 0.5329137444496155, 0.055189669132232666, 1.7746740579605103, 2.048480272293091, -0.7363224029541016, 0.39272844791412354, 1.3962831497192383, 0.6009083986282349, -0.4533385932445526, 0.9171431660652161, 0.4598414897918701, -0.7424697279930115, -0.15992611646652222, 1.0305912494659424, 0.009865342639386654, -0.8660557866096497, 0.17236824333667755, -0.9594717025756836, -0.20848621428012848, 0.34834024310112, 0.7900484204292297, -0.9325469136238098, 1.2992998361587524, 0.043910156935453415, -1.2886344194412231, 1.579017996788025, 0.41266560554504395, -0.4540368318557739, 0.14089912176132202, 0.29666757583618164, 1.022475242614746, -0.27647921442985535, -1.0419235229492188, 0.6847141981124878, -0.5299985408782959, -0.7945975661277771, -0.9251127243041992, -0.02189231663942337, -1.8715354204177856, -0.03083653934299946, -0.1934177428483963, -0.8846598863601685, 0.03834189847111702, -1.5586899518966675, -0.44032180309295654, 0.828304648399353, 0.010897482745349407, 0.7361892461776733, 0.7327755093574524, -1.5969849824905396, 2.0387589931488037, 0.5153277516365051, 0.007545149885118008, 0.32267212867736816, -0.12462395429611206, 0.3307781517505646, 2.357820749282837, -0.6650218963623047, -1.0735242366790771, 0.23823963105678558, 0.935880184173584, -0.4868338108062744, -0.29820767045021057, 1.158456802368164, -1.086811900138855, -1.5261118412017822, 0.19560645520687103, 1.4348747730255127, 1.6083683967590332, -1.6067450046539307, 0.32846736907958984, 0.9581345915794373, -0.6939622759819031, -1.1283313035964966, 1.5984907150268555, -0.45786046981811523, 1.145750880241394, -0.7497455477714539, 0.49166229367256165, -0.51055908203125, -0.6740090847015381, -0.48384737968444824, 0.4369269013404846, -1.0628225803375244, -1.550328016281128, 1.1883561611175537, -1.5283974409103394, 0.7455746531486511, -1.156973958015442, -0.6542581915855408, 1.1487360000610352, 2.5982487201690674, -0.35203802585601807, 1.43978750705719, -0.1756618469953537, -1.2804558277130127, -0.6107550859451294, -0.9502449035644531, 0.231392502784729, -0.2697519361972809, -1.1509662866592407, 0.5033571124076843, -1.3077398538589478, -1.0228497982025146, 0.1401996612548828, -0.22453603148460388, 1.41214120388031, -1.8599121570587158, 1.116044521331787, -0.46086981892585754, -1.0644015073776245, 1.4500179290771484, -0.7609441876411438, 1.8477762937545776, 0.1112852543592453, -0.00587882474064827, -0.10362186282873154, 0.8568546175956726, 0.05962761491537094, -0.09492428600788116, -1.1316025257110596, -0.34488916397094727, 1.1640431880950928, -0.5225270986557007, -0.8949933052062988, -1.50780189037323, -0.166104257106781, -1.6370927095413208, -1.4153971672058105, -0.5309544801712036, -3.2983765602111816, -0.39535388350486755, 0.3203440308570862, 0.9664721488952637, 1.0827970504760742, 0.8965966105461121, 1.7413594722747803, 1.0529526472091675, 0.2160893827676773, -0.6014415621757507, 0.16450399160385132, 0.9856210350990295, -1.3237032890319824, -1.6982851028442383, 0.5257108807563782, 0.6545108556747437, 1.3815261125564575, 0.03497391194105148, 0.6910752654075623, 1.3068486452102661, 0.27149146795272827, 1.1244175434112549, -0.9129254817962646, -1.2443703413009644, 1.1529661417007446, -0.3162083029747009, 0.8734174966812134, 1.0096707344055176, 0.7842432260513306, 1.8324819803237915, 1.2280192375183105, 1.1099343299865723, -0.3523786664009094, 0.23126327991485596, -0.20627905428409576, -1.4869532585144043, -0.8098887205123901, 0.728773295879364, -0.7682176232337952, -0.33218246698379517, -0.44633960723876953, -0.2939714789390564, -0.9807749390602112, -0.6912676692008972, 1.24486243724823, 0.05793187394738197, -0.4083233177661896, 1.1122819185256958, 0.24325627088546753, 1.7842583656311035, 1.4653501510620117, -1.4020448923110962, -0.14237269759178162, -1.8303396701812744, 0.6807995438575745, 0.915806233882904, -0.41957154870033264, -0.7236407995223999, -0.2896978259086609, 0.10677550733089447, -0.215846985578537, -0.3359147608280182, 1.3348129987716675, -0.5114983320236206, 0.9082621335983276, 0.6356105208396912, 0.20004552602767944, 0.03622129186987877, -0.3449460566043854, -0.22857093811035156, 0.5459105968475342, -0.6580929756164551, -2.932020902633667, -1.684175729751587, -0.557746410369873, -0.05728698894381523, 0.22385849058628082, 1.2724286317825317, 0.4461196959018707, 0.917437732219696, 0.3752526044845581, -0.07133062928915024, 1.524588704109192, 0.48895397782325745, 0.6462002396583557, -0.603512167930603, -0.5398502349853516, 1.9021660089492798, -1.243219017982483, 0.33776697516441345, 0.38080114126205444, 1.067551612854004, -0.793134331703186, -1.2448570728302002, 2.095141649246216, -1.128261923789978, -0.8079499006271362, -1.8026589155197144, 0.22592173516750336, -0.9007469415664673, 1.3446036577224731, -1.2895952463150024, -0.19717611372470856, 0.3415471613407135, -1.55527663230896, 1.2295671701431274, 0.776081919670105, 0.9071297645568848, -0.9772348403930664, -0.5201790928840637, -2.330836057662964, -0.07261676341295242, 0.5631358623504639, 0.08574296534061432, 1.2347725629806519, 0.6770386695861816, -0.6460530757904053, 0.49480587244033813, -0.6625688076019287, -0.18893979489803314, 1.445918321609497, -0.012158949859440327, 0.2103530764579773, 0.005362147465348244, 0.5245261788368225, -0.14005541801452637, 1.1496998071670532, -1.6259725093841553, 0.6007373332977295, 0.6910330653190613, 0.7788126468658447, -0.010937219485640526, -0.28148290514945984, 1.4530186653137207, -2.26798939704895, -0.021793626248836517, 0.3732200562953949, -0.2155674695968628, 0.7764477729797363, 1.047648310661316, 0.16528040170669556, -1.7760567665100098, -0.3646194636821747, -0.05669349431991577, -0.12245695292949677, 0.2165566384792328, 2.2323317527770996, -0.5374783873558044, -1.5842602252960205, -0.7706084847450256, 0.749654233455658, 0.5330646634101868, 0.546328067779541, 0.30492866039276123, -0.6811784505844116, -0.7481154799461365, 0.30955109000205994, -0.11634130030870438, -0.15253576636314392, 2.3214120864868164, 1.3028831481933594, -0.2909467816352844, 0.060252584517002106, 1.2807234525680542, 0.8070332407951355, -0.7235946655273438, 1.1053725481033325, -0.2446897029876709, 1.7700871229171753, -0.3716573119163513, 0.7398631572723389, -2.087912082672119, -0.616332471370697, 0.1864505112171173, 0.30926844477653503, -1.0633221864700317, -2.610653877258301, 1.1968718767166138, 0.5484455823898315, -1.5706032514572144, 1.551567554473877, 0.3382519781589508, 0.1457308828830719, -0.9324570298194885, -0.46356815099716187, 0.5585673451423645, -0.3003116548061371, 0.14469915628433228, 1.2686890363693237, -0.2053142488002777, 1.392686128616333, -0.7535483241081238, 0.3385773003101349, -2.7130308151245117, 1.5872000455856323, -2.0383975505828857, 0.15885433554649353, -0.03190477937459946, 0.22160640358924866, -0.8499715924263, -0.8411165475845337, -2.0730645656585693, 1.561519742012024, -2.0051844120025635, 1.4741235971450806, -1.923445463180542, -0.6111800074577332, 1.0248761177062988, -0.4401908218860626, -0.9421129822731018, 1.6868879795074463, -0.642353892326355, -0.3904592990875244, -1.3047879934310913, 2.1227824687957764, -0.6792796850204468, -1.322556972503662, -0.5591263771057129, -0.5370513796806335, 0.33394312858581543, 1.0499701499938965, 0.22440555691719055, 1.1563977003097534, -0.5239948630332947, -0.6502143740653992, -1.500927209854126, -2.5236918926239014, 1.4057669639587402, 0.8227888345718384, -0.2291545569896698, -0.19650687277317047, -0.5286865830421448, 1.1243079900741577, 1.1348382234573364, -0.9537934064865112, 0.4036499857902527, -0.4063533544540405, -1.9888094663619995, 1.0905104875564575, 2.031942844390869, 0.19851240515708923, -0.8334108591079712, 0.7729117274284363, 0.3048839569091797, -1.2693865299224854, 0.1363096833229065, 1.2799590826034546, -1.731104850769043, -1.1330193281173706, -0.9401248693466187, 0.39277708530426025, -0.17138491570949554, 0.08451805263757706, 0.7055537700653076, 0.475179523229599, 1.3372297286987305, 0.7340437173843384, -1.421644926071167, -1.5368318557739258, -0.033207401633262634, -0.3949737548828125, 0.4111711382865906, 0.3788015842437744, 0.13571062684059143, -0.039233241230249405, 1.0715444087982178, -0.6761778593063354, -0.6197831630706787, 1.1251994371414185, 1.065127968788147, -0.8677033185958862, 0.11560986936092377, -0.25077635049819946, -0.9175944924354553, 1.964797019958496, 0.04093821719288826, 0.05589748173952103, 0.48850440979003906, -0.2107686549425125, -0.21974803507328033, -0.5180936455726624, -0.6784409880638123, 1.0358424186706543, -0.7311586737632751, -0.7385176420211792, -1.2046180963516235, 0.36528685688972473, 1.341010570526123, -0.7405047416687012, -0.14162664115428925, 1.8474947214126587, 1.4955946207046509, -1.0162616968154907, 0.102350614964962, -0.6632657051086426, 0.6741530895233154, 0.3907468318939209, -0.36387842893600464, -0.6161503195762634, -0.6613957285881042, 0.2044762670993805, -0.16113559901714325, 9.592126298230141e-05, -0.08031147718429565, 0.5930745601654053, -3.1832823753356934, -0.5996822714805603, -1.2097748517990112, 0.040413204580545425, 0.13710284233093262, -1.0166993141174316, -0.7671874165534973, -0.5583217144012451, 2.2414352893829346, 0.245534747838974, -1.5607234239578247, -2.302381753921509, 0.537990152835846, -1.041014552116394, -0.40522581338882446, -1.4333398342132568, -0.5852554440498352, 0.12941287457942963, 0.452123761177063, -0.5930238962173462, -2.957714319229126, 0.11279609054327011, -0.7957687973976135, 0.6991668343544006, -0.250132292509079, -1.2696865797042847, -0.6320575475692749, 1.5240235328674316, 0.2251690775156021, -0.8405260443687439, -0.29763656854629517, 0.9122682809829712, 1.3934199810028076, 0.6142202615737915, -1.811685562133789, -1.5096503496170044, 0.4316428303718567, -0.1216193363070488, -0.4782451391220093, -1.0336973667144775, -1.4922387599945068, 1.3348186016082764, -0.7001340985298157, -0.8491829633712769, 1.8621306419372559, -0.29938745498657227, 0.8569689393043518, -0.5237699151039124, 1.1414974927902222, 1.3062176704406738, 0.2822049856185913, -0.27006763219833374, 0.9923970699310303, 0.7448753714561462, 0.0510793998837471, 0.2652316689491272, -0.03894132003188133, -0.38331329822540283, 0.1597486138343811, 0.07105298340320587, 1.303297996520996, 0.7789475321769714, 0.898878276348114, -0.42542460560798645, 1.1479017734527588, -1.9989724159240723, -1.5767558813095093, -0.14565643668174744, -0.1313750147819519, 0.7901374101638794, -0.030732115730643272, -1.0285218954086304, -1.6692852973937988, 0.42010438442230225, -0.8558669090270996, 1.4607723951339722, -0.2844007611274719, 0.39422550797462463, 0.5007140040397644, 0.2639792859554291, 1.3101682662963867, -0.8448883891105652, 0.44054877758026123, -1.1307578086853027, 1.886697769165039, -1.49053955078125, -1.0575127601623535, 0.5378714799880981, 0.6777728199958801, 0.028335275128483772, -0.5070764422416687, -0.6214993000030518, 0.9893372058868408, -1.5724077224731445, 0.13616864383220673, 1.531181812286377, 0.27526065707206726, 0.42807018756866455, -1.916007161140442, -0.5320053100585938, 0.37750470638275146, 1.6420506238937378, -1.2375175952911377, 0.4912366271018982, -1.510926604270935, -0.9508277177810669, 0.21834757924079895, -1.0940215587615967, -1.1536303758621216, 0.7256041765213013, -0.7260973453521729, -0.5592241883277893, -0.19682231545448303, 0.4641974866390228, 0.3742571473121643, -1.0166478157043457, -0.9696577787399292, 1.3275035619735718, -0.021073631942272186, 1.0876890420913696, -0.010408055037260056, 1.665063738822937, -0.7245998382568359, -0.9797762036323547, -0.9751908183097839, 0.8251808881759644, 0.7328009605407715], [-0.5443810224533081, -0.38619303703308105, -1.747190237045288, 1.0491411685943604, -0.17054679989814758, 0.3979029655456543, 2.0640807151794434, 0.09952422976493835, -0.786339282989502, 1.5189400911331177, -0.761614978313446, 0.528904914855957, -0.5249089002609253, 0.9300864934921265, -0.6491060256958008, -0.26368263363838196, -1.306124210357666, 0.6905098557472229, 0.38444486260414124, -0.07224684953689575, -0.12851054966449738, -3.227264404296875, 0.20356903970241547, -0.13021278381347656, -0.043055981397628784, 1.8201594352722168, 1.6279239654541016, -1.5552449226379395, 0.844238817691803, -0.6208365559577942, 1.3016008138656616, -0.5558523535728455, -1.4725356101989746, 0.7843976616859436, 0.5745290517807007, -2.6640264987945557, 0.9326625466346741, -0.08620628714561462, -1.0038460493087769, -0.4246881306171417, -0.2246263474225998, -0.22688592970371246, -0.3508482873439789, -0.21405203640460968, 0.7709159255027771, -2.6807219982147217, -0.1979980319738388, 0.48319530487060547, -1.3723262548446655, 0.03098674677312374, -1.1192346811294556, 0.5819825530052185, -0.8353478312492371, 0.4328477680683136, 1.2274059057235718, -0.08326734602451324, 1.8758454322814941, -0.822685182094574, -0.12001807242631912, 0.4758094847202301, 1.2176299095153809, -1.1472011804580688, -0.010277020744979382, 1.3464634418487549, -1.6287471055984497, -1.0101336240768433, -0.6017258763313293, 0.951272189617157, -0.6083120703697205, -1.4254531860351562, 1.2102558612823486, -0.5861738324165344, -0.7724551558494568, 1.211613655090332, -0.5581104755401611, -0.6917885541915894, 1.5755338668823242, 0.40627002716064453, -1.601158618927002, 0.8855431079864502, -2.137637138366699, -1.5853192806243896, -0.9402028918266296, 0.7161862850189209, 0.5152705907821655, 0.8081699013710022, 0.7521493434906006, 0.11153866350650787, -0.006770718842744827, 0.010281550697982311, -0.794185996055603, 1.5595591068267822, 0.5551732182502747, 0.28972071409225464, 0.08607195317745209, 0.5712692737579346, -0.713998019695282, -1.324188232421875, -0.2708897590637207, -0.6145829558372498, -0.5940575003623962, -1.1868263483047485, 0.8910009860992432, 0.5473461151123047, 0.1578870713710785, 2.474672794342041, 0.32116180658340454, 0.9345824122428894, -1.638770341873169, 0.6158254742622375, -0.629231870174408, 1.2441070079803467, -1.0691773891448975, 0.18038448691368103, -1.1110894680023193, -0.6589498519897461, 0.48939406871795654, -0.7195302844047546, 0.25483477115631104, -0.18431170284748077, -0.667822539806366, 0.7002686858177185, 0.1218881756067276, -1.1084765195846558, 1.0747233629226685, -0.18021106719970703, 0.5565313100814819, 1.0151907205581665, -0.8462492227554321, -0.3720565140247345, -0.7377679347991943, 0.03136250749230385, 0.9769096970558167, -0.5303187966346741, -1.339388370513916, -0.03676435723900795, -0.34014892578125, 0.5287728309631348, -0.4128785729408264, 0.31041550636291504, 0.42208850383758545, -0.4206831157207489, -0.794079601764679, 0.4282303750514984, 0.9578481912612915, 0.7791988849639893, -0.47524237632751465, -0.8777526021003723, 0.22527487576007843, -0.8984492421150208, -0.5427701473236084, -0.3076384365558624, -0.3270014524459839, -0.5443111062049866, -1.0432603359222412, -0.3683064579963684, 2.21065092086792, -1.064710021018982, -1.443766713142395, -0.03269745036959648, -0.7084785103797913, 1.172859787940979, 0.2821689546108246, 1.5684406757354736, 0.33425867557525635, 0.9409196376800537, 0.17798125743865967, -1.4020392894744873, -0.697946310043335, -0.25618162751197815, -1.9009754657745361, 0.09910478442907333, 0.9514791369438171, 1.5558136701583862, -1.4652734994888306, -1.2162749767303467, 1.3780683279037476, -1.3458213806152344, 1.0627827644348145, -0.975894570350647, 0.27610069513320923, -0.15312449634075165, 1.8912882804870605, -1.5210171937942505, -0.552859902381897, 0.7205355763435364, -1.659700870513916, 0.08239936828613281, 0.08018771559000015, 0.4466094970703125, -1.3526238203048706, -0.19712361693382263, -0.18344800174236298, -0.2644951641559601, 1.4532185792922974, -0.3049534261226654, -0.6989664435386658, -0.29329580068588257, 0.7682220935821533, 0.05148039013147354, -0.8041234016418457, -0.7646924257278442, 0.5391100645065308, 0.4683274030685425, -1.008455514907837, -0.29065361618995667, -1.2692631483078003, -0.598026692867279, 0.9938651323318481, -1.6522321701049805, -2.014779567718506, 2.0578272342681885, 1.0782667398452759, -1.117944598197937, 0.2979363799095154, -0.14930836856365204, -1.092952013015747, 0.14388927817344666, 1.032194972038269, -0.5756087303161621, 0.5368586182594299, -2.2416961193084717, 0.5077094435691833, 0.5067773461341858, 1.5075798034667969, 0.8625526428222656, 0.5770309567451477, 0.7120799422264099, 0.8909841179847717, -0.731696367263794, -0.7373098134994507, -0.20069636404514313, 1.0354336500167847, -0.24474681913852692, -0.40364089608192444, 1.3052902221679688, 1.0889259576797485, 0.4821787476539612, -0.37898072600364685, -0.5285837650299072, 1.1217565536499023, -0.47287610173225403, 0.9787466526031494, 0.06993450224399567, 1.2083399295806885, 0.42595961689949036, 1.4229716062545776, -1.3512884378433228, -0.7748170495033264, -0.4546394944190979, -0.9830520749092102, -0.3558374345302582, 1.3143266439437866, 1.41966712474823, -1.15095055103302, -1.1713920831680298, -0.5533924102783203, -0.12669934332370758, -0.06413168460130692, -1.3519545793533325, 0.6557375192642212, 0.12161038815975189, -0.34392476081848145, -0.36289742588996887, 1.3099786043167114, -0.36373528838157654, 0.7339475750923157, 1.815974473953247, 1.3154410123825073, -0.4658658802509308, 0.1481282114982605, 0.7232595086097717, 0.7364376783370972, -1.9827035665512085, -0.34503889083862305, 0.9446430206298828, -1.3149100542068481, -0.26682472229003906, -0.3604646921157837, -0.7446063756942749, -0.4534810781478882, -0.20434147119522095, -0.7702043056488037, 0.029058435931801796, 1.126187801361084, 0.25876349210739136, -3.3986761569976807, 0.18113940954208374, 0.05770288407802582, 0.05057201534509659, -0.1025145873427391, 0.3065217435359955, -0.3037067651748657, 0.12340477854013443, 0.018150322139263153, -0.3211575746536255, -0.9962083697319031, -1.5567878484725952, 0.3123546838760376, -0.050308749079704285, -0.45150691270828247, -0.49451667070388794, -0.19328634440898895, -0.7359176278114319, 0.44503024220466614, -1.0269842147827148, 0.24650061130523682, 0.014691631309688091, 0.7333553433418274, 0.4550057649612427, 1.2575536966323853, -0.08927890658378601, -1.848048448562622, -1.154486060142517, -2.2699637413024902, 1.6937099695205688, -0.4582558870315552, -0.938029408454895, 2.0435383319854736, -0.42384782433509827, 1.828518033027649, -0.2973511219024658, -1.140081763267517, -0.47951218485832214, -0.35282450914382935, 0.5802704095840454, -0.7763248682022095, 1.0411908626556396, 1.1704150438308716, -0.21397271752357483, 0.00392781151458621, -0.1796746551990509, 0.7357852458953857, -0.2466263473033905, -0.40296316146850586, -0.30655744671821594, 1.1030828952789307, -0.7018551826477051, 1.2316358089447021, 2.18534255027771, 1.325869083404541, -0.9236912131309509, -0.1369543820619583, -0.2288265973329544, -1.4345735311508179, -0.9688639640808105, -0.5532114505767822, -1.900191068649292, -0.9400596022605896, 0.5615178942680359, -0.7390145063400269, -0.09919757395982742, 0.9092167019844055, 0.04115516319870949, 1.2999240159988403, -1.6115431785583496, -0.5463264584541321, 0.2975536286830902, 0.5423524379730225, -1.3972258567810059, -1.76712167263031, 0.2078845202922821, 0.2845102548599243, -0.8429535627365112, -0.2573758661746979, -1.0473687648773193, 0.2981451749801636, 0.0036979748401790857, -0.6855287551879883, -0.023606112226843834, 1.6690131425857544, 0.6663103699684143, -0.48677700757980347, -0.44882023334503174, 1.828481912612915, 1.2591688632965088, 1.9519374370574951, -1.4328346252441406, -1.4701279401779175, -0.022688429802656174, -1.5311163663864136, -0.5310848951339722, 0.859504759311676, 0.4202824831008911, -0.5450021624565125, 0.612188458442688, -0.3764723241329193, -2.026515245437622, 0.8734087944030762, -0.9764055609703064, -0.03953583911061287, -0.000734595931135118, -2.4991872310638428, 0.2954423725605011, -0.7037397623062134, -0.3706313371658325, 1.1799323558807373, -1.3117157220840454, 0.2105429470539093, 0.096185602247715, -0.02051735296845436, 0.9747757911682129, -0.7877948880195618, -1.940852403640747, -0.865324854850769, -0.23269692063331604, -1.2451268434524536, -0.7577333450317383, -0.3524538576602936, -0.9675211310386658, -0.938188910484314, 1.4232455492019653, -0.9571698307991028, 0.6395042538642883, -0.2865314483642578, -2.412665605545044, -0.8506655693054199, 1.4897087812423706, 2.5090811252593994, -0.4458411931991577, -1.0300754308700562, -0.7736523747444153, 0.6410757899284363, -0.6212717294692993, 0.5405950546264648, 0.42235538363456726, -0.806565523147583, 0.46425482630729675, 0.7863919734954834, 1.325947880744934, -1.422942042350769, 0.5024271607398987, -1.129936695098877, 1.2803608179092407, -0.7011731266975403, -1.9087486267089844, -0.5706350207328796, 0.8804160952568054, 0.6884005069732666, -0.7108151912689209, 0.8681291937828064, -0.3301515579223633, 0.058780886232852936, -1.5487180948257446, 0.5665250420570374, 2.080155849456787, -0.8438475728034973, -0.9782093167304993, -0.05497703328728676, -0.31006884574890137, -0.03532062843441963, 0.05642767250537872, 0.8978300094604492, 0.2619362771511078, 0.5151593685150146, 0.6402755379676819, 0.6000176668167114, 0.3518023192882538, -0.5359465479850769, 0.10674791783094406, -0.47595736384391785, 0.6389931440353394, 0.2965278625488281, -0.929536759853363, 1.8791987895965576, -1.7941038608551025, -0.20050369203090668, 0.1505904346704483, -0.9023887515068054, 1.1393460035324097, 0.5040730237960815, 1.5365301370620728, 1.4143308401107788, 0.7243019938468933, 0.45435377955436707, -0.727249801158905, -0.7419324517250061, 1.2690017223358154, 2.6298670768737793, 0.41593289375305176, -0.7767947912216187, 0.7710448503494263, 1.552409291267395, -2.683621644973755, 0.38248544931411743, -0.4850698411464691, 1.6651614904403687, 1.5813792943954468, 0.6061957478523254, -0.308059960603714, 0.30724668502807617, 0.891266942024231, 0.4027944803237915, 0.23812632262706757, 0.1497478187084198, -0.5057108402252197, -0.8243019580841064, 0.681467592716217, -1.1011427640914917, -1.6807478666305542, 0.26744309067726135, -0.43110111355781555, 0.07819321006536484, 1.2379531860351562, -0.053436946123838425, 0.6662393808364868, -0.2853532135486603, 0.34316486120224, 1.0864224433898926, 1.6985291242599487, 1.8986928462982178, -0.7537845373153687, -1.5699313879013062, -0.6901222467422485, -1.7748329639434814, -1.8837637901306152, -0.28582531213760376, -0.9406598806381226, 0.40743833780288696, -0.1919342577457428, 1.2215090990066528, 0.4763586223125458, -0.5488802790641785, 0.24421779811382294, 0.3750636875629425, -1.2144970893859863, 1.0803178548812866, -1.6162115335464478, -0.20583756268024445, -1.5988439321517944, -1.943406105041504, 1.3076419830322266, -1.5222173929214478, 1.6150894165039062, 1.2767648696899414, -0.8582737445831299, 0.8555861711502075, -1.1355056762695312, -1.9127706289291382, 1.7723602056503296, 1.1568095684051514, 0.2605699300765991, 0.042161229997873306, 1.3776925802230835, -0.4728219211101532, 0.8275907039642334, 0.15625488758087158, 0.3249536156654358, 0.3335265815258026, -0.10492526739835739, -1.7118257284164429, -0.9210246801376343, -0.27258777618408203, -0.545526921749115, -0.07801211625337601, -2.098696708679199, -0.43460118770599365, -0.576163113117218, 0.5433967709541321, -0.3328729569911957, -0.1362595111131668, -1.2482253313064575, -0.4133121967315674, -0.3562484383583069, 2.704900026321411, 1.3995662927627563, -1.3730138540267944, -0.0029078179504722357, -0.587479293346405, 0.4069942533969879, -0.3290242552757263, 0.9520502686500549, -0.8132182955741882, -0.8361022472381592, 0.8331736326217651, 1.2079334259033203, -2.074899673461914, -0.4760676920413971, 0.2351023554801941, -2.114753246307373, 0.46442118287086487, -0.40469858050346375, -1.6293421983718872, 0.18372707068920135, -0.17780043184757233, -0.48803552985191345, 1.0296396017074585, -0.5484941005706787, 0.11242537200450897, -0.42526358366012573, 1.8356506824493408, -0.41259586811065674, 0.7539292573928833, 1.6836172342300415, 0.6646388173103333, -0.6367712020874023, -0.3250015676021576, -0.9516042470932007, -2.02687931060791, -0.6186761260032654, -0.8044852614402771, -1.4777144193649292, -1.1565030813217163, -0.39552390575408936, 0.28235021233558655, 2.578310966491699, 1.2685587406158447, 0.5082423686981201, -0.017369044944643974, -1.159091830253601, 0.687828540802002, 2.4241061210632324, 1.4812674522399902, -1.7872166633605957, 0.02044609934091568, 0.3027975261211395, -1.309959888458252, -1.077530026435852, 1.1209830045700073, 1.1170032024383545, 0.34103551506996155, 0.04644124582409859, -0.1481224149465561, 0.7527828216552734, 0.24361386895179749, 0.38334426283836365, -0.9648550748825073, -1.3258378505706787, -1.4948532581329346, -1.106417179107666, -0.5234972238540649, -0.8882958889007568, -0.565897524356842, -0.28584423661231995, 0.5129861831665039, -0.10530104488134384, -0.2498624175786972, 1.2003554105758667, 0.1705925166606903, 1.6533032655715942, -0.8203492760658264, -0.9237428307533264, 1.2506396770477295, 0.39999669790267944, 0.08309942483901978, 0.5178712606430054, 0.8186492323875427, 0.6512516140937805, -0.10678238421678543, 1.1601558923721313, -1.3053398132324219, 0.5831595659255981, 0.3155333399772644, -0.6757909655570984, 0.20508767664432526, 0.21928851306438446, 1.3103488683700562, 1.3403210639953613, 0.9709917902946472, -0.858834981918335, -1.2161381244659424, 1.2023152112960815, -3.0797736644744873, 0.027810577303171158, -0.5480231046676636, 0.19064050912857056, 0.29902687668800354, 0.0003221736114937812, -0.23839586973190308, -1.1594302654266357, -1.7385796308517456, 1.1836397647857666, 0.023036610335111618, -0.7987731695175171, -1.13030207157135, 2.0860064029693604, 1.197490930557251, -0.35245099663734436, -0.35300955176353455, -0.7120811343193054, 1.1997215747833252, 0.591502845287323, -0.9119833707809448, 1.3081461191177368, 0.021493423730134964, 0.3606976866722107, -0.5644233822822571, -0.6187870502471924, 0.9854779839515686, -0.8266201019287109, -0.2522505223751068, -2.2273945808410645, -0.4135909974575043, 0.642956554889679, 1.268041729927063, -1.4263426065444946, 0.2720884382724762, 2.3148295879364014, -1.002762794494629, 0.4557607173919678, -1.2071409225463867, -0.8142052292823792, -0.46982434391975403, 0.582162082195282, 1.8884122371673584, 0.6089552044868469, -0.522535502910614, 1.0101522207260132, -0.2440907508134842, 0.2570408582687378, -1.0939949750900269, 0.06078615412116051, -0.21344314515590668, 1.2775967121124268, -0.48523980379104614, -1.8173013925552368, 1.7755200862884521, -0.5705766081809998, -0.3327120244503021, 0.5967473983764648, 0.9459961652755737, 1.506110429763794, -0.406887412071228, 0.31869879364967346, -0.08784261345863342, -0.1916084587574005, 0.2788931131362915, 0.2388923168182373, -0.8853975534439087, -0.7298593521118164, -0.3998548090457916, -0.4882986843585968, -0.3063870072364807, 1.8503392934799194, -0.91315096616745, -1.0700130462646484, -0.5016205906867981, 0.4608633518218994, -0.4624227285385132, 0.17839749157428741, -0.011389173567295074, 0.6523171663284302, -0.2686404883861542, 0.08761626482009888, 0.13311821222305298, -0.721638023853302, -0.06042392924427986, 1.316385269165039, 0.04297882318496704, 0.8431885242462158, -0.37576019763946533, -2.7043943405151367, -0.5702115893363953, 1.4413409233093262, -0.25705474615097046, 0.29758647084236145, -0.4134167432785034, -0.17412106692790985, -0.4311659336090088, 0.014774647541344166, -0.9306020736694336, -0.533939003944397, 0.29104137420654297, 0.925820529460907, 0.9804061651229858, -0.1633753478527069, -1.2726114988327026, -2.5296099185943604, -0.38020747900009155, -1.121420979499817, -1.047956943511963, 2.042945623397827, -1.1178559064865112, -1.1720798015594482, -2.131972312927246, 0.6634410619735718, 2.822746515274048, -1.4546374082565308, -0.4538849890232086, -0.5677045583724976, 0.658636748790741, -1.1503039598464966, -1.4856431484222412, 2.8765883445739746, 1.0585532188415527, 0.0007824280764907598, 1.2775975465774536, -0.026172220706939697, -0.5208625197410583, -0.7606527209281921, -1.701363205909729, -0.8444327116012573, 0.016994593665003777, 0.07107819616794586, -0.5660793781280518, 0.7384462952613831, 0.3457562327384949, -0.031256068497896194, 0.39206963777542114, 0.204594224691391, 1.8402066230773926, -0.10954087227582932, 0.9601932168006897, -0.9672961235046387, -1.5749198198318481, 0.4622504711151123, -1.8286058902740479, 1.4253039360046387, 0.7759898900985718, -0.6397294998168945, 0.016214897856116295, -1.5512727499008179, 0.4192556142807007, 2.1373822689056396, 0.8259103298187256, 1.112243890762329, -0.296489953994751, -0.30702391266822815, 1.07620108127594, 1.1589500904083252, 1.280902624130249, -2.180820941925049, 0.042406801134347916, -0.2956826984882355, 1.9559342861175537, -1.2852529287338257, 0.04433716833591461, -1.0421721935272217, 1.4017629623413086, 0.37286216020584106, -0.024829350411891937, -0.5746732950210571, 1.4132438898086548, -0.43783700466156006, -0.37590888142585754, 0.9181071519851685, 0.46355390548706055, -0.5229488015174866, 1.480865240097046, -1.1205888986587524, 0.7528833150863647, 1.5506224632263184, -0.5561504364013672, 1.2650058269500732, -1.015491008758545, -1.2039886713027954, -1.5634832382202148, -1.6541097164154053, 0.6202355623245239, -0.0060658445581793785, -0.47709301114082336, -1.538604736328125, 0.46495798230171204, -0.2553671896457672, -2.200742483139038, -0.24700215458869934, -0.2591160535812378, 1.0387219190597534, -0.4746766984462738, 0.6902751922607422, 0.2530417740345001, 0.4626387059688568, 0.05129384621977806, -1.0189553499221802, 0.03884492814540863, 0.48340171575546265, 1.1179912090301514, 0.35821712017059326, -0.9058725833892822, 0.15043415129184723, 0.5132089257240295, -0.957329511642456, 0.1448753923177719, -2.5083565711975098, 0.19375742971897125, -0.012935657054185867, 1.0084550380706787, 1.0392870903015137, 0.6224251389503479, -0.006273915525525808, -0.12505044043064117, 1.1497832536697388, -1.2352317571640015, -0.8641979098320007, 1.9607774019241333, -0.5907562375068665, -1.1117228269577026, -0.21998274326324463, -0.04447504132986069, 1.0003323554992676, 1.8671753406524658, -0.2454681247472763, 0.5471881628036499, -0.2969549000263214, -0.29757651686668396, 0.9193184971809387, 0.485819935798645, -1.1034291982650757, -0.3460513949394226, 0.5664119720458984, -0.7850285768508911, 0.40914440155029297, -1.4664089679718018, -0.7912569642066956, -1.1572197675704956, 0.6299745440483093, -1.4632532596588135, 0.8058810830116272, -0.698773205280304, -1.7208932638168335, -1.0014091730117798, 0.4356781244277954, 0.2619311213493347, 0.6071984767913818, 0.2660177946090698, 1.1574147939682007, 2.2766170501708984, 0.850529670715332, 0.2691892683506012, 0.5399847626686096, 0.07660798728466034, 0.15819863975048065, -0.2702641785144806, -0.3653582036495209, 0.6640539765357971, -0.5624600052833557, -0.4874671995639801, -2.2571523189544678, 0.4929068982601166, -1.4524247646331787, -2.5365235805511475, 1.4137119054794312, -0.69301837682724, -0.16482482850551605, -0.6581594944000244, -0.20145951211452484, 1.5163590908050537, -1.124609351158142, -1.1295201778411865, -0.030399933457374573, 0.23397889733314514, -1.0274624824523926, -1.1207760572433472, 0.47538837790489197, -0.6160070300102234, -0.22694671154022217, 0.22621075809001923, 1.134823203086853, 0.9599345922470093, -0.7224076390266418, -0.1686934530735016, -0.3815305233001709, -1.6816728115081787, -0.8953536748886108, 2.1948206424713135, 0.598080039024353, 1.5847485065460205, 0.1280253529548645, 0.873555600643158, -0.39052829146385193, -1.7235949039459229, -1.1619268655776978, 1.0327214002609253, -1.160738229751587, 0.5291389226913452, -0.539853036403656, -0.405798077583313, -0.7094837427139282, -0.832274317741394, -0.9018295407295227, -1.7912135124206543, -1.3655472993850708, 1.0319408178329468, 0.023890143260359764, -0.1401352733373642, 0.3940895199775696, -1.2129508256912231, -0.4909234344959259, -0.3527071177959442, -0.5686127543449402, 0.30706140398979187, 0.08572226762771606, -0.47304344177246094, -1.6951813697814941, -0.20511628687381744, 1.1884685754776, 0.29620763659477234, -0.7159329056739807, -1.5829442739486694, 1.557544469833374, 1.123732089996338, -1.952496886253357, 0.7813272476196289, -0.4602406620979309, -1.784389853477478, 0.7584397792816162, 0.6343781352043152, -0.977533221244812, 0.7160558700561523, -0.15030333399772644, -0.4297211170196533, 1.251899242401123, 1.0743036270141602, 0.8735436201095581, 0.08584639430046082, -3.0301427841186523, 0.30719342827796936, -0.8249063491821289, -0.5532912611961365, 2.3604068756103516, -0.6824284195899963, -0.02222212590277195, 1.4833102226257324, -0.8792001605033875, -1.210633635520935, -0.9106633067131042, 0.13272452354431152, -1.3981519937515259, 0.9705966114997864, -0.47007107734680176, 0.9941319227218628, 0.7932877540588379, -0.9950664043426514, 1.2054722309112549], [1.0345253944396973, -0.8157828450202942, 0.3305339515209198, 1.857274055480957, -0.1009252592921257, 0.3180043399333954, -0.9672044515609741, 0.48647090792655945, -0.10992967337369919, -0.2657172381877899, -1.523887038230896, 0.31762242317199707, -0.6247684955596924, -2.4711108207702637, 0.595055341720581, 0.15235501527786255, -1.4196338653564453, -1.080740213394165, 0.39920923113822937, 2.374696731567383, -0.3799910247325897, 0.7901107668876648, -0.0026419467758387327, 1.350304365158081, -0.8968973159790039, 0.0660770907998085, -0.22440531849861145, 0.10113009810447693, -0.8534409999847412, -0.8397052884101868, -1.3073384761810303, -0.18649715185165405, -1.0425148010253906, -0.1726817488670349, 0.13833899796009064, 0.11533248424530029, -0.5953494310379028, -1.5334868431091309, 0.5502016544342041, 2.286637544631958, 0.04986603185534477, 2.319430112838745, -1.3627140522003174, 0.8731592893600464, 0.8549597263336182, -0.8666238188743591, -0.16062556207180023, 0.09376658499240875, -2.427825689315796, -0.654821515083313, 0.596120297908783, -1.5030983686447144, 0.7254828810691833, -0.28335830569267273, -0.7129192352294922, 1.5417287349700928, 0.8667364716529846, 0.24343274533748627, 0.12629437446594238, 0.641270101070404, 0.3321579694747925, 0.28675904870033264, -0.951885461807251, 0.12782230973243713, 0.8149368166923523, -0.06854798644781113, 0.3759329617023468, 0.011766423471271992, 0.48078805208206177, 0.539014995098114, 0.6919337511062622, -0.4551975727081299, -1.0340955257415771, -0.8136234879493713, -0.5775591731071472, -0.01317258644849062, 0.3590942323207855, 1.776097059249878, -0.625576913356781, 0.6067851185798645, -0.005188502836972475, -0.9752476215362549, 1.0412423610687256, 2.566981554031372, -0.8475192785263062, -1.536068320274353, -0.1821148544549942, -0.2301190197467804, -0.2554256319999695, 0.2878011465072632, -0.5670950412750244, 1.5402718782424927, -1.2469360828399658, -0.8213263154029846, -0.670855700969696, -0.004806190263479948, 0.08755611628293991, -0.3274904191493988, 0.13403527438640594, -0.30419933795928955, -0.6314030289649963, -1.6843103170394897, 0.8064314126968384, -0.5135622024536133, -0.09518537670373917, 0.9449188709259033, 0.738585889339447, 0.29562342166900635, -1.2492032051086426, -0.2066563218832016, 0.5696008205413818, 0.2556668817996979, -0.27986279129981995, 0.7585429549217224, -0.6949195861816406, 0.3022664189338684, 1.350281834602356, -0.6273000836372375, 0.5361033082008362, 1.19864821434021, -0.3272145092487335, 0.2294193059206009, -0.12657999992370605, -1.2543145418167114, 1.080419898033142, 0.10889382660388947, 0.5504049062728882, -0.16934734582901, 1.2280901670455933, 0.817274272441864, -0.7223294377326965, -1.168796181678772, 1.0336464643478394, -0.21397238969802856, -0.7499167323112488, 1.018561840057373, -1.2850046157836914, -1.8214564323425293, -0.20774750411510468, -0.150589719414711, -1.7016288042068481, 0.7258625030517578, 0.9637482166290283, -1.7721059322357178, 0.5035629272460938, -0.13131466507911682, -0.17704960703849792, -0.8543968200683594, 1.6516304016113281, 0.3529016673564911, 0.11904313415288925, 0.011733259074389935, -0.12729604542255402, -0.4791480600833893, 0.039968617260456085, 0.38644111156463623, -0.8440258502960205, 0.371915340423584, -0.9261916279792786, -2.016876220703125, -0.15538160502910614, 0.7644586563110352, 1.004042148590088, -1.052834391593933, -0.989398717880249, -0.13966087996959686, -1.4942232370376587, -0.4011644721031189, -2.4435665607452393, 0.2126573622226715, 1.5890957117080688, -0.7051658034324646, 1.0331496000289917, 0.908413827419281, -1.2641005516052246, -0.2436302751302719, 0.6661044359207153, 0.6121867299079895, -0.6498150825500488, -1.1585696935653687, -1.4878497123718262, 0.09323006123304367, -0.46743524074554443, -0.3496477007865906, -0.38739603757858276, -0.3390912115573883, 0.5036448836326599, 1.619907021522522, -0.7121910452842712, -0.6967178583145142, 0.4383420944213867, -0.9530835151672363, -1.2885637283325195, -0.2627357542514801, -0.21572963893413544, -0.16431796550750732, -1.0596517324447632, 1.4230797290802002, 0.5067325234413147, -0.0363222137093544, 1.0741219520568848, -0.35817021131515503, 0.014298350550234318, -0.7922883629798889, 0.36637040972709656, -1.1126466989517212, -1.5757313966751099, 0.5664047002792358, 0.5859852433204651, 1.0323294401168823, 0.08110908418893814, -0.026223765686154366, -0.40043866634368896, -0.35590672492980957, 0.1026146337389946, 0.8501893281936646, -0.5351810455322266, -1.8113949298858643, -2.2075259685516357, 0.1480809897184372, -1.113715410232544, 1.8077659606933594, 0.16225722432136536, -1.687455415725708, -0.7987112998962402, -1.585151195526123, -1.0680897235870361, 0.4401567876338959, 1.362201452255249, -1.6597644090652466, -0.5795698165893555, -0.32720911502838135, -0.1361524909734726, -0.9965456128120422, -1.073459267616272, 0.15099424123764038, -2.6941027641296387, -0.9832978248596191, -0.4123969078063965, 1.6613786220550537, 0.5107532143592834, -0.3898163437843323, 0.35566282272338867, -2.491472005844116, -0.16395802795886993, -0.3418228328227997, 0.862741231918335, -1.6768079996109009, 0.030261898413300514, 0.01017370168119669, -1.028910517692566, -0.20074628293514252, -0.6289390921592712, 0.48695364594459534, 0.9137980937957764, 1.3916583061218262, -0.2586645483970642, -2.1465327739715576, -0.08373809605836868, -0.011057382449507713, 0.8422600626945496, -0.5427537560462952, -1.1012274026870728, 0.32079967856407166, 0.21865740418434143, -1.0828899145126343, -0.627964198589325, 0.3759462237358093, -2.2729837894439697, 0.28905877470970154, 1.3275872468948364, 0.136297345161438, 0.30901017785072327, 0.7082061767578125, 0.28956836462020874, 0.5804641246795654, -0.12017074972391129, 0.5468063354492188, 0.4085015654563904, 1.2929542064666748, 0.48552823066711426, 0.005094422493129969, 0.4574059247970581, 0.3576131761074066, -0.4172075390815735, -0.36598098278045654, -0.8658431172370911, 1.063969373703003, 0.7453871965408325, 0.4249919652938843, 1.2648043632507324, -0.25224584341049194, 0.23528845608234406, -0.6370486617088318, 0.13029494881629944, 1.5824862718582153, -0.054239992052316666, -0.07021307945251465, 0.35508352518081665, -1.6978496313095093, 1.4012482166290283, -1.1157994270324707, -0.7110828161239624, -1.0699269771575928, -0.13031786680221558, 0.8973162174224854, -0.5355767011642456, -1.2984023094177246, 0.7351944446563721, -0.7248885631561279, -0.1894330084323883, -0.7108504772186279, 0.7667312026023865, 2.5926756858825684, -0.09682877361774445, 0.05890878662467003, -0.9018698930740356, 0.4814832806587219, -0.11905943602323532, -2.355666160583496, 0.6650325655937195, 1.410672664642334, 0.6979156732559204, -1.8834229707717896, 2.3755550384521484, 0.5662961006164551, 0.37384507060050964, -1.405035138130188, 1.579573631286621, 2.0292253494262695, -0.3536151051521301, 1.3026520013809204, 0.6665404438972473, 1.554699182510376, 0.1399141252040863, -0.21894225478172302, -0.4048510491847992, 0.6233687996864319, -0.7231733202934265, 0.016490748152136803, 0.10665462166070938, 1.5390890836715698, -1.7284936904907227, 0.47652125358581543, 0.12773621082305908, -0.9117786884307861, -0.8690392374992371, 0.6480890512466431, -0.06514551490545273, -0.4483862519264221, 0.8215345740318298, 0.7169556617736816, 1.2832257747650146, -2.1732094287872314, -0.4478851854801178, 0.06034662947058678, -0.3151453733444214, 0.9291531443595886, 0.6708023548126221, 0.47288787364959717, -0.6547811627388, 0.8538286089897156, 0.6868575811386108, 1.4954020977020264, -0.08085223287343979, -0.07460074871778488, 1.2812420129776, -0.068931445479393, -0.5342864990234375, -0.09118099510669708, -0.11335961520671844, 1.684433102607727, 0.4480350613594055, 0.6599617004394531, 0.08946036547422409, 0.05157075449824333, 0.486595094203949, -0.24473822116851807, 0.2316218912601471, 1.2235429286956787, 1.4599828720092773, -1.6033880710601807, 1.3203669786453247, 0.9989424347877502, 0.7080535888671875, -1.3155577182769775, -1.0859984159469604, -1.3251005411148071, 0.3428994119167328, 0.4570620656013489, 1.685088872909546, -1.10332190990448, 0.9921296834945679, 0.07230644673109055, 0.0892513319849968, -1.6086698770523071, 0.5264653563499451, -0.0032984830904752016, 0.8678204417228699, -0.3436126410961151, -1.1088167428970337, -1.3394393920898438, 0.1648288369178772, 2.1491012573242188, -0.47553882002830505, -0.5976073741912842, 0.6811550259590149, -2.5281314849853516, -0.4510231018066406, -1.6417063474655151, -0.03509789705276489, 0.8908600211143494, -0.10537809133529663, 0.6852127313613892, -0.7695982456207275, -1.111625075340271, -0.9211626052856445, -2.150470018386841, 1.2015628814697266, -0.516081690788269, -0.7761167287826538, 1.262300729751587, 0.5895172357559204, 1.3074672222137451, -0.9771922826766968, -1.9906814098358154, 0.5032329559326172, -1.078782081604004, -0.3735094666481018, 1.563378930091858, -0.5641363859176636, -1.1011717319488525, -0.30414098501205444, -0.7949290871620178, 0.6231089234352112, -0.7286693453788757, -2.4688050746917725, 0.5780664682388306, -0.43373578786849976, -0.6075152158737183, -0.2818293869495392, -0.1849643737077713, -0.4298611879348755, 1.1799912452697754, 0.6565237641334534, 0.6526818871498108, 0.5837764739990234, 0.3980589807033539, 0.45102518796920776, 0.7532497644424438, -1.596788763999939, -0.06576544046401978, -2.018448829650879, 1.048414945602417, -1.1587539911270142, -0.510631799697876, 2.8105013370513916, -0.2621583044528961, 0.36831045150756836, -0.10199576616287231, -1.2825771570205688, 0.320363312959671, 0.8720579743385315, -0.6037313938140869, -1.4793105125427246, -0.2670964300632477, 2.052215099334717, 1.930582880973816, 0.6371229887008667, -1.5106394290924072, 0.034207239747047424, 1.2913480997085571, 0.07314705848693848, 0.40242311358451843, -0.2983538508415222, -0.2679154872894287, 0.6725752949714661, -0.9414904117584229, 0.11084041744470596, -0.9077522158622742, 1.2066378593444824, -1.239968180656433, 0.3568478226661682, 1.055274486541748, 2.385890245437622, -0.47963380813598633, 0.09354927390813828, -1.207759976387024, -0.4663717448711395, -1.1923052072525024, -0.954251229763031, -0.8028848171234131, -1.7132320404052734, 0.12767474353313446, 1.093380093574524, 1.3725404739379883, -0.19148272275924683, 1.602426528930664, -0.6023856401443481, -0.6445491909980774, -0.3270183205604553, 1.0438439846038818, 0.14138925075531006, 1.5046645402908325, 0.43394380807876587, 1.2188609838485718, -1.1212390661239624, 0.9312195777893066, 1.6421817541122437, -0.6867290139198303, -1.6160657405853271, 0.7077325582504272, 1.2230225801467896, -0.5057260990142822, -0.5406729578971863, 0.4666554033756256, -0.2298554629087448, -0.8639671802520752, 0.8457186818122864, -0.5930112600326538, 0.7281639575958252, -0.4522801637649536, -0.6883254051208496, -1.2695262432098389, 0.15269359946250916, -0.8363581895828247, -1.0176104307174683, 0.49143779277801514, 0.32336246967315674, -1.0648819208145142, -2.2644453048706055, -1.1595557928085327, -0.6281795501708984, 2.180842161178589, -1.2134385108947754, 0.9540974497795105, -0.29489749670028687, 0.4393906593322754, -1.1497759819030762, 0.44330278038978577, -1.4583021402359009, -1.9724972248077393, -0.4494747221469879, -0.33526480197906494, 0.7968034744262695, 0.8023498058319092, 0.7830772995948792, -2.2105295658111572, 0.11102528870105743, 1.135349154472351, 0.47928377985954285, -0.5639622807502747, 0.7148629426956177, 2.6043972969055176, 0.16326503455638885, -0.4635840654373169, -0.005527557339519262, 0.6052269339561462, -1.4553649425506592, 0.8348551392555237, -0.6070309281349182, -0.32223063707351685, 0.8511614799499512, -1.0262725353240967, 1.543616533279419, 2.6963388919830322, 0.8792435526847839, 1.4148731231689453, -0.18388143181800842, 2.0106866359710693, 0.3278753161430359, 0.6616122722625732, -0.7145711183547974, 2.0796444416046143, -0.5865058302879333, 0.025282103568315506, 0.2786198556423187, -1.8805763721466064, -0.0012457713019102812, 1.1174930334091187, -0.6130802631378174, -0.39987823367118835, -0.6578565835952759, -1.0558239221572876, -1.307236909866333, -0.33863380551338196, 0.6519705653190613, -1.2855422496795654, -0.024678613990545273, -1.5734080076217651, -0.9025057554244995, -0.1211371198296547, -0.2931744456291199, -1.1786905527114868, -1.6267447471618652, 0.3981882929801941, 0.1728619486093521, 0.1756879985332489, -0.9897301197052002, 0.3172819912433624, -0.27590712904930115, 1.3332880735397339, 0.4231451153755188, -0.987866997718811, -0.2583293318748474, 0.09010884910821915, 2.205106735229492, -0.8179401755332947, 0.1771576851606369, -1.2268530130386353, -0.23047907650470734, 0.8269004821777344, 0.15077154338359833, -0.824368953704834, 1.8257744312286377, 1.1352145671844482, -0.47828808426856995, -0.9864763617515564, 0.007696032989770174, 0.467058926820755, -1.866278886795044, -1.1973309516906738, -0.7464570999145508, -0.15128915011882782, 0.2123054414987564, 0.9091443419456482, 0.5889244079589844, -0.5339949131011963, -1.64826500415802, 0.9202941060066223, -2.5677590370178223, 0.9540599584579468, 2.8966450691223145, 0.22130483388900757, -0.24368968605995178, -0.7717590928077698, 0.9851136803627014, 1.4786311388015747, -0.6703474521636963, 2.2086853981018066, 1.252455472946167, -0.858171284198761, 1.5768893957138062, 0.8019749522209167, 0.3835211396217346, 1.220657467842102, -0.12546001374721527, -0.8695964813232422, -1.3960667848587036, -0.8827203512191772, 1.0980716943740845, 0.5861953496932983, -1.0710426568984985, 1.4005992412567139, -1.3130872249603271, 1.1727861166000366, -1.1311858892440796, -2.383415460586548, 0.4290846586227417, 1.2659406661987305, 1.9465492963790894, 1.304796576499939, 1.6924145221710205, -0.524222195148468, -0.24492216110229492, -1.0872985124588013, 0.5669453144073486, 1.3051578998565674, 2.1926934719085693, 1.349317193031311, 0.13207979500293732, 1.2627718448638916, -0.7888385057449341, 0.4823434352874756, -0.3386913239955902, 0.8110145926475525, 0.4285750687122345, 0.0756494402885437, 1.3433830738067627, 0.49491816759109497, 0.37622448801994324, 0.8616632223129272, 1.088618278503418, 1.2267929315567017, -0.8126479983329773, 1.1380901336669922, 0.8993948101997375, 1.3159607648849487, -0.5018272399902344, 1.3831360340118408, 0.1834535449743271, -1.089511513710022, 0.7719176411628723, -1.1355178356170654, -0.4027540683746338, -0.29143765568733215, -0.3791697323322296, 0.12342660874128342, 0.705202579498291, -1.0811614990234375, -1.9204213619232178, -0.023862328380346298, 2.120565414428711, -0.0008714979630894959, -1.0552343130111694, 0.7195751070976257, -0.9665378928184509, 2.1031293869018555, 1.655152440071106, -0.7024769186973572, 0.418632447719574, -2.0354244709014893, -0.014765960164368153, 0.190717414021492, -2.2217442989349365, -0.09355655312538147, 1.1423245668411255, -0.6264861822128296, 1.2466806173324585, -0.055073630064725876, -2.185450792312622, -0.16715238988399506, -0.7008837461471558, 0.9848257899284363, -0.49577638506889343, 0.5627845525741577, -0.4667273759841919, 0.7803456783294678, 0.03817065805196762, 0.8947901129722595, 0.711097776889801, 0.7232373356819153, 0.6015160083770752, -0.48828473687171936, -0.6222169399261475, -0.6193229556083679, 0.09350833296775818, 0.8539360165596008, 0.6406445503234863, 1.269665241241455, -0.39455169439315796, -0.12541820108890533, 0.7062122225761414, 1.0544167757034302, -1.302551031112671, 1.0543041229248047, -1.4450418949127197, 0.26244601607322693, 0.7138932943344116, -2.7565817832946777, -0.09401565790176392, 1.5471140146255493, -1.624547004699707, -0.490478515625, 0.7256308197975159, -0.08652283996343613, -0.3385181128978729, 1.4616187810897827, 0.777177095413208, -0.06827782839536667, 0.7580256462097168, 0.7454556226730347, -0.8133018016815186, 0.4325369596481323, -0.1344132274389267, 0.029464421793818474, -2.30098819732666, 0.07731910794973373, 0.5981269478797913, -0.43106502294540405, 0.43801313638687134, 1.8576451539993286, -0.49154528975486755, -2.11128568649292, -0.8812310695648193, 0.6545947194099426, -1.1995266675949097, 0.8838773369789124, -0.9681777954101562, -1.4811224937438965, 0.320983350276947, 0.8345292806625366, -0.24232929944992065, 0.6777801513671875, 1.363211750984192, -1.189803957939148, -1.003296136856079, 0.90740966796875, -1.2033416032791138, -0.46094632148742676, -0.21701225638389587, -0.5725660920143127, 0.7059362530708313, -0.03731892630457878, 0.6367843747138977, -0.9639592170715332, 1.250244379043579, 0.39768892526626587, -1.0599970817565918, 1.9001212120056152, 0.5955917835235596, -1.3302668333053589, -0.38212457299232483, -0.42813727259635925, -1.2003955841064453, -0.8735463619232178, -0.25275859236717224, -1.2011573314666748, -1.553902268409729, -0.8062810301780701, -0.8282414078712463, -0.27980783581733704, 0.8701499104499817, -0.6142959594726562, 1.318276286125183, -1.4460667371749878, -0.22914063930511475, 0.49233511090278625, 0.25985220074653625, -1.8028409481048584, -0.35958197712898254, 1.8867429494857788, -0.07080043852329254, -0.696426510810852, 0.007654116954654455, -0.48077160120010376, -0.1530463844537735, -0.36416196823120117, -0.14153483510017395, -0.8295788764953613, 0.5072563290596008, -0.5600149631500244, 0.39253321290016174, 0.9929889440536499, 1.7430272102355957, 0.5981619358062744, 0.4856596291065216, -0.5456519722938538, 0.9377487301826477, 0.03559804707765579, 1.079818606376648, -0.38807612657546997, -0.7058042287826538, -0.8720494508743286, 0.16235750913619995, -1.6328167915344238, 0.9135271906852722, 0.7709687352180481, 0.35224953293800354, 0.2367299348115921, 0.8123873472213745, 0.2583502233028412, -0.14496169984340668, -0.5081027746200562, -0.27731019258499146, 1.281374216079712, 0.14245979487895966, -0.6158206462860107, -0.026078661903738976, 1.0799628496170044, 0.8202675580978394, 0.03740834817290306, 2.7476859092712402, -0.4831616282463074, 1.4954423904418945, -0.21676233410835266, -1.663108468055725, -0.14265897870063782, 0.6060101389884949, 0.3144816756248474, 0.5613002181053162, -0.543194055557251, -0.9086428880691528, -0.2676906883716583, 1.4302000999450684, -1.824422001838684, 0.41119176149368286, 0.1896849274635315, -0.3707369565963745, -0.6617557406425476, -1.3885561227798462, 1.01558256149292, -1.3365304470062256, 1.297021746635437, -0.23624490201473236, 0.519511878490448, 0.20308014750480652, 1.4512253999710083, 0.48764291405677795, -0.9898350834846497, -1.6492785215377808, 0.8773667216300964, 0.84380704164505, 0.09026610106229782, 0.2649129033088684, -1.3966940641403198, 1.309649109840393, -1.5541465282440186, -0.03886641934514046, 0.8722895979881287, 1.333817481994629, -0.04089973494410515, 0.5861928462982178, 2.0820631980895996, -1.2381426095962524, 0.13834148645401, -0.5002269744873047, -1.0328657627105713, -0.2132631540298462, 0.12720981240272522, 0.09538809955120087, 0.8821586966514587, 1.263442873954773, 1.4967944622039795, -0.13015903532505035, -0.34623658657073975, 0.916434109210968, -1.3586523532867432, -2.04669451713562, 1.269431710243225, 1.2063387632369995, 0.9579740762710571, -0.11410945653915405, -0.3136273920536041, 0.2865631878376007, -0.7402396202087402, 0.8375192880630493, 1.0860449075698853, 0.514240562915802, 0.5101478695869446, -0.790912389755249, 0.00021912246302235872, -0.880772054195404, 0.8323386907577515, -0.963371753692627, 0.1950034648180008, -1.1541242599487305, -0.21358895301818848, 1.4050941467285156, 1.7216891050338745, 1.4138174057006836, -0.2697904706001282, -0.44000834226608276, 1.0388667583465576, -0.42749330401420593, -0.5146411657333374, -0.6497898101806641, 0.7171813249588013, -0.16360355913639069, -0.769512951374054, -0.8594580292701721, 0.4137745499610901, 1.4194777011871338, 0.7489591240882874, -0.13397018611431122, -0.5443390607833862, -1.5817692279815674, 0.20726194977760315, 0.6965615153312683, 1.1938546895980835, -2.1058754920959473, 0.6379218697547913, -0.6573671698570251, -0.2891687750816345, -0.2715403139591217, -1.5898780822753906, 2.252915620803833, -0.3862006962299347, 1.069614291191101, -0.41956496238708496, 1.231695294380188, 1.5169519186019897, 1.025670051574707, -0.9115926623344421, -0.3191920816898346, -0.9074999690055847, -0.8498225212097168, -2.7728631496429443, -0.9280208945274353, 0.5072935223579407, 0.1678103804588318, 0.40757137537002563, 0.21318921446800232, 1.4241982698440552, -0.7412499189376831, -0.705045759677887, -0.16371583938598633, -1.8168518543243408, -0.4714217782020569, 1.5530513525009155, 0.26554766297340393, -1.1689192056655884, 0.48643958568573, -1.5622289180755615, -1.4543391466140747, 1.353384017944336, -1.1103849411010742, -0.07220298796892166, 2.1250247955322266, -0.45443376898765564, -0.743118166923523, 0.8285871148109436, -0.787675678730011, 1.3730838298797607, 1.1687718629837036, -2.373396396636963, 0.7524318099021912, 1.2974296808242798, 0.59403395652771, 0.7460384368896484, 0.23055852949619293, 0.9397149085998535, 0.10965711623430252, -0.38199108839035034, 1.9784942865371704, 0.3309485614299774, 0.6898852586746216, 0.43171223998069763, 3.055647134780884, -0.012077444233000278, -1.0488132238388062, -0.12425250560045242, -1.7405214309692383, 0.7589141130447388], [-0.018641099333763123, -0.04827093333005905, -1.4927735328674316, 0.5316712856292725, 1.213956594467163, 1.478956699371338, 2.393204927444458, 3.8874568939208984, 0.20804107189178467, 0.38199281692504883, -1.5391778945922852, 0.7509018778800964, 0.26028403639793396, 0.3979617655277252, 0.7783832550048828, -0.5767496824264526, -1.4877288341522217, 1.027691125869751, -0.6349404454231262, 0.6708477139472961, -0.32497820258140564, 2.1655635833740234, -1.079872727394104, -0.9564330577850342, 0.6590656638145447, -0.612845242023468, -0.20237822830677032, 0.4953582286834717, -1.6439545154571533, -0.6824145317077637, 1.1360656023025513, 0.8974835276603699, -1.799471378326416, -1.5319476127624512, 1.393393635749817, -0.8049766421318054, 0.554310142993927, -0.6611892580986023, 0.1830216646194458, -0.1584625393152237, -0.8163236975669861, 3.014024257659912, 1.4344438314437866, -0.7388558983802795, 1.0963445901870728, -0.31981202960014343, 0.4589390456676483, 0.026254119351506233, 0.41431155800819397, -1.7244653701782227, 0.605435311794281, -1.8180359601974487, -0.028908701613545418, -0.3533326983451843, 0.8762096166610718, -0.032220687717199326, 0.4381941556930542, 0.6545889377593994, 0.8955951929092407, -0.6037356853485107, 1.7886399030685425, -1.239285945892334, -0.11900240182876587, -0.2711898982524872, -0.7487293481826782, 0.866311252117157, -0.7170301079750061, 1.2425750494003296, -1.8239437341690063, 0.09632433950901031, 0.7786388397216797, 1.2047827243804932, -0.4710579514503479, 0.4633050858974457, -0.14321398735046387, 0.34578582644462585, -2.0077250003814697, 0.2451034039258957, 0.6770279407501221, 0.6062899231910706, 0.2664380371570587, -0.41158339381217957, -0.8842218518257141, 1.039185881614685, 1.0789991617202759, 0.2653402090072632, -0.649456799030304, 1.3058724403381348, -1.9544622898101807, 1.5465948581695557, -0.35551363229751587, -0.5541898012161255, -0.37686586380004883, 0.8389281034469604, -0.05519920215010643, -0.346587598323822, -2.827106237411499, -2.8674535751342773, -1.1245219707489014, 0.1974055916070938, 0.09204016625881195, 1.8434410095214844, 0.43039825558662415, 0.30213505029678345, -0.8714808225631714, 1.2497085332870483, 0.09381524473428726, -0.17758788168430328, 1.2334420680999756, 0.942365825176239, 1.853474736213684, -0.8683128952980042, 0.5164691805839539, -0.579003632068634, -1.4753714799880981, -1.1207672357559204, -0.02090870589017868, 0.945359468460083, -0.4970690906047821, 1.4953835010528564, -0.5428953170776367, -0.9592812061309814, 0.5707662105560303, -0.5983902812004089, -0.007060030475258827, -0.4647713601589203, 0.03421706333756447, 0.6712045669555664, -0.06738238781690598, 0.9531400799751282, -0.5753063559532166, 0.9185017943382263, -1.6680331230163574, -0.28804051876068115, 0.15343330800533295, -0.8496376872062683, 0.3365330994129181, 0.1700373888015747, 0.6758437156677246, -0.9033448100090027, -1.4323183298110962, -0.3073669970035553, -2.2835288047790527, -0.041810210794210434, 2.4699044227600098, 0.2581362724304199, -0.6656341552734375, 0.7871548533439636, 0.7852419018745422, 0.8146126866340637, -1.5672640800476074, 0.04470817744731903, 0.8917667865753174, 0.07356125861406326, -0.19284169375896454, 1.657637119293213, 0.2050183117389679, 0.6583985686302185, 1.4859668016433716, -0.4449200928211212, -0.36089444160461426, 1.0412336587905884, 0.6795119643211365, 0.2037934958934784, 1.3011088371276855, 2.327077865600586, 1.0440970659255981, 1.8683068752288818, -1.6045774221420288, 0.7637429237365723, -0.9591622352600098, 0.7655209302902222, -0.6158389449119568, -1.8015071153640747, 0.2718610465526581, 1.6299926042556763, -0.9100199937820435, 0.36307424306869507, -1.2070876359939575, -0.2466040700674057, 1.1543248891830444, -0.4178522825241089, 0.7336993217468262, 1.051684021949768, 0.05294271558523178, -0.11883701384067535, 0.07499849051237106, -0.40808308124542236, -0.27229270339012146, -0.28007787466049194, -0.3339725732803345, 0.15324877202510834, 1.5194114446640015, 0.004378842189908028, 1.3498384952545166, 1.7126569747924805, -0.2827015817165375, -0.45628082752227783, -0.5218969583511353, -0.3843797743320465, 0.6294827461242676, -0.12183400243520737, -0.11640807241201401, -0.1662237048149109, 0.798194169998169, 0.7376900315284729, 0.2053976207971573, 1.2194050550460815, 0.7107024192810059, 0.5528019666671753, 0.869433581829071, 0.458511620759964, -1.1077420711517334, -0.9974772334098816, 1.7008230686187744, -0.30536308884620667, 0.38435930013656616, -0.5246638655662537, -2.36745285987854, 0.11089061200618744, -0.7678092122077942, -0.08220045268535614, 0.46408870816230774, -0.896973729133606, 0.463652640581131, 0.30777356028556824, -1.4486744403839111, 0.30263587832450867, -0.0697489008307457, 0.7498688101768494, 1.5604238510131836, -1.3700847625732422, -1.3107450008392334, -0.3462529182434082, -0.27784228324890137, -0.007581162732094526, -0.440738320350647, -1.0116077661514282, -0.20052184164524078, -1.3190886974334717, -1.4010963439941406, -1.830756425857544, -0.013699558563530445, 0.06970886141061783, 0.2825714349746704, -0.5408322811126709, -0.36197420954704285, 0.2788214087486267, -0.9765750169754028, -1.1098052263259888, 0.24117332696914673, -0.519305944442749, 0.31505367159843445, 1.170392394065857, 1.0552691221237183, -1.4926228523254395, 1.1819723844528198, 0.16083501279354095, 1.3841036558151245, 0.7223149538040161, -0.426911324262619, 0.5428500175476074, -0.7879483103752136, 0.07635543495416641, 0.5677962899208069, -0.7233185172080994, -0.18730275332927704, -0.12915760278701782, 0.17886269092559814, -1.1090813875198364, -1.2862073183059692, -0.30684515833854675, -0.8708696961402893, 1.0396629571914673, -0.28387367725372314, 0.6768819689750671, 0.43890267610549927, 0.8943454027175903, 0.5488871932029724, 0.2512109577655792, -2.286280393600464, -1.546329379081726, -1.3567477464675903, 0.9305784702301025, -0.41149166226387024, 0.5980509519577026, -0.08113317936658859, 1.0930801630020142, -0.942802906036377, -0.4379798173904419, 1.0520702600479126, -0.5864688754081726, -0.48402902483940125, 0.5158271789550781, -0.10137973725795746, -1.5973570346832275, 0.2936355173587799, -4.045675277709961, -0.7826457023620605, -1.9734960794448853, 1.3377519845962524, -0.3168429434299469, 0.4677203595638275, 0.38339829444885254, -1.400223970413208, 0.5116264820098877, 0.4557781517505646, -0.5090323090553284, 0.8627932667732239, -0.7197979688644409, -0.24340103566646576, 0.2573590874671936, -0.9776583909988403, -1.7593533992767334, -2.0073297023773193, -0.884375810623169, -2.3007519245147705, -1.2755186557769775, -0.9700186252593994, 1.839210867881775, 0.7115511298179626, -1.6282438039779663, 1.342640995979309, 2.841771125793457, -2.1745896339416504, -0.9066452980041504, -0.10519637167453766, 0.48160597681999207, 0.6607443690299988, -2.2242558002471924, 0.01910742186009884, 0.21719419956207275, -0.5120428800582886, -0.03577724099159241, 1.2293545007705688, 0.7386597394943237, 0.800695538520813, -1.8674546480178833, 0.9922146201133728, -0.2809556722640991, 0.2735348343849182, 1.2030845880508423, -0.5632265210151672, 0.30327120423316956, 1.7562575340270996, 1.9737813472747803, 0.41398248076438904, -0.042176514863967896, 1.862161636352539, -1.0659326314926147, 0.46845459938049316, -0.3652329444885254, -0.028860965743660927, 1.357649564743042, 0.5149499773979187, -0.04351024329662323, 0.4511657953262329, -0.3992084264755249, -0.3239063024520874, -0.41172105073928833, 2.23416805267334, -0.028195912018418312, -0.9682857990264893, -1.5379525423049927, -0.8115794062614441, -0.8056686520576477, 2.1121582984924316, -0.07215049117803574, 0.45934247970581055, 0.2004871517419815, -0.7274667024612427, 0.3349807858467102, -0.8081775903701782, 1.8668071031570435, 1.1447099447250366, 0.6913508772850037, 1.3998541831970215, -1.4050425291061401, 1.1579222679138184, 0.35737141966819763, -0.15163418650627136, 0.9446519017219543, 0.5028543472290039, -1.5032844543457031, -0.22624163329601288, -1.4639296531677246, 0.0778575986623764, 0.11616102606058121, -1.4244385957717896, 1.0326722860336304, 1.8819317817687988, 1.0109009742736816, -0.7338300347328186, 1.5419763326644897, 0.26653623580932617, 1.0118635892868042, -1.0863351821899414, -0.5837176442146301, -0.11983511596918106, 2.265381336212158, -0.36705315113067627, 1.4561012983322144, -0.6760464906692505, -0.08708234876394272, -0.9443759918212891, 1.5460022687911987, -0.6055096387863159, 0.5989713668823242, 0.7965587973594666, 1.489622950553894, 0.02576441317796707, 0.2874918282032013, -1.5441384315490723, -0.1442856639623642, -2.1452183723449707, -0.39782604575157166, 0.13170450925827026, -1.6058156490325928, -0.6794253587722778, -1.1377898454666138, 1.0258917808532715, -0.7876287698745728, -1.4138166904449463, 1.4654335975646973, -1.481479525566101, 1.5013923645019531, -0.5435805320739746, 0.9467533230781555, -0.38660693168640137, -1.0362573862075806, -0.19544246792793274, -1.4286532402038574, 0.4296419024467468, 1.0819629430770874, 0.44299569725990295, -0.3144559860229492, -1.0817642211914062, -2.371838331222534, -1.0318214893341064, -0.2200666218996048, 0.012923255562782288, 1.1459661722183228, -1.745447039604187, 0.163955956697464, -0.4830024540424347, -0.5996686816215515, 1.8159867525100708, 1.0198335647583008, -0.5814363360404968, -0.46670687198638916, -1.0761247873306274, 1.6057394742965698, -0.8313986659049988, 0.16365721821784973, 0.1896759271621704, 0.48567524552345276, -0.03566638380289078, -0.5644635558128357, -0.9460150599479675, 0.4394558370113373, -0.6689324378967285, 2.3324337005615234, 0.616716742515564, -0.5043410062789917, 0.9773008227348328, -0.1432763785123825, 0.04874049499630928, -0.6524462103843689, -1.078393816947937, 0.5674892067909241, -0.4752805233001709, -1.0157934427261353, -0.3703218996524811, 0.8978328704833984, -0.01023915782570839, 1.3223552703857422, 0.20690016448497772, -1.8557995557785034, -0.7808599472045898, -0.25557994842529297, 0.7537006139755249, 0.040824320167303085, -0.7470439076423645, 0.3903898000717163, 1.4052778482437134, 0.3407796025276184, 0.1933404803276062, 0.22702878713607788, -0.21656683087348938, 0.3685292899608612, 1.6319682598114014, 0.16003528237342834, -1.038256049156189, 1.6928303241729736, -0.14329320192337036, -0.7294780015945435, -0.18475547432899475, -1.7968770265579224, -0.7815641760826111, 0.39866334199905396, -0.17708715796470642, -0.6412449479103088, 0.4321339428424835, -0.39070966839790344, -0.7485114932060242, -0.022897275164723396, 1.1172444820404053, 2.397008180618286, 0.10288866609334946, -0.09443417936563492, 0.19748979806900024, 0.5212337374687195, 0.1102990210056305, 1.260818600654602, 1.0363500118255615, -0.6717259287834167, -0.09395012259483337, 1.5442157983779907, -1.6121466159820557, 0.7313315272331238, 0.4605388045310974, -0.7438803911209106, 0.5036274790763855, 0.6585203409194946, 0.8073161244392395, 0.2247084379196167, 0.09154578298330307, 0.07446833699941635, -0.26127174496650696, -0.14619852602481842, 0.5455427169799805, -0.24327585101127625, 0.08451510220766068, -1.3059418201446533, -0.7632176876068115, -1.3158870935440063, -0.12074720114469528, 0.7363876700401306, -2.8730931282043457, 1.2922502756118774, 1.0972126722335815, 0.6015650629997253, 0.8449685573577881, 1.097388505935669, -0.6871613264083862, 2.0780158042907715, -0.4334419369697571, 0.9471976161003113, 0.8758024573326111, -1.3799958229064941, 0.45897138118743896, 0.2979748547077179, -0.9192095398902893, -0.006871390622109175, -1.7246404886245728, 2.7402496337890625, -1.0292834043502808, 0.4891089200973511, 0.3996790945529938, 0.899976909160614, -1.0875041484832764, -1.8007917404174805, 1.0057623386383057, 0.7465479373931885, 0.3791491389274597, 0.009198191575706005, 0.005972520913928747, 0.653376042842865, -0.3140234053134918, -0.25334104895591736, -0.13130366802215576, 0.9150665998458862, -2.209982395172119, -1.0181353092193604, -1.2111151218414307, 0.7100838422775269, 0.2080003172159195, -0.278104692697525, -0.5548716187477112, 1.4192038774490356, -0.16609631478786469, -0.6362109780311584, 0.4526708722114563, -0.251662939786911, 0.8004290461540222, -0.6417554616928101, -1.7787415981292725, 0.15443842113018036, 2.9454267024993896, -0.5064054131507874, -0.3984900712966919, -0.45406192541122437, -0.7150416374206543, 0.23944693803787231, -0.12838417291641235, -0.46689265966415405, 0.8125815391540527, 0.1661626398563385, 1.5930083990097046, 2.226050853729248, -0.9573696255683899, 0.12965121865272522, -0.04236730933189392, -1.120737910270691, 1.4966739416122437, 0.18708550930023193, -0.7116976976394653, -0.42031824588775635, -1.408157229423523, -2.986186981201172, -0.07587658613920212, 1.7478201389312744, 0.9371817111968994, 0.7564855813980103, -0.15577998757362366, 0.3502482771873474, -2.093104124069214, -0.2689412832260132, 0.5907979011535645, -1.163092017173767, 0.9448474645614624, 1.8419991731643677, 0.2360863834619522, -0.003797694807872176, -2.028905153274536, -0.29590946435928345, 1.3633607625961304, 2.4833805561065674, 0.12186791747808456, -0.7550241351127625, -0.26200830936431885, 0.46109437942504883, 1.0703357458114624, 1.4010075330734253, 1.5362796783447266, 1.194819450378418, 1.3318525552749634, 1.7687060832977295, 1.083763837814331, 1.4566981792449951, 0.5957027673721313, 0.4187544584274292, -1.5211236476898193, -0.6979262828826904, -0.29898062348365784, -1.3601701259613037, 1.2409182786941528, 0.3953431248664856, 1.7201340198516846, -0.20793019235134125, 1.0443229675292969, -0.34342896938323975, 0.5537379384040833, 0.05749514326453209, -0.32230082154273987, 0.2334921807050705, 1.2535886764526367, -1.109374761581421, -0.03439868614077568, 0.07305853068828583, -1.6817224025726318, -1.2048845291137695, 0.12378045171499252, -0.9029572606086731, -0.847161054611206, -0.0764438807964325, 0.135794535279274, 1.5942224264144897, -1.5218771696090698, -0.7519763708114624, 0.9455395340919495, 0.2065885365009308, 1.458263635635376, -2.053007125854492, -2.1026275157928467, -0.06074219197034836, 0.5005218386650085, 0.14941249787807465, -0.9932234883308411, -0.8993929624557495, 0.6749899983406067, -0.38058996200561523, 0.5868484377861023, -0.13408207893371582, 0.5025860667228699, -0.29286959767341614, 1.668093204498291, 0.5344241261482239, -2.0764691829681396, 0.36351558566093445, 0.5261673331260681, 0.04207971692085266, -1.1304199695587158, -0.2659207880496979, -2.063873767852783, 0.10254430770874023, -0.22573590278625488, 1.330689787864685, 0.056153956800699234, 1.5799130201339722, 0.5003534555435181, 0.6696578860282898, -0.5068058371543884, 0.1236824095249176, 0.2610384225845337, -0.2571466267108917, 0.7700945138931274, -0.6749232411384583, -0.5902947187423706, 1.5945446491241455, 0.9173480868339539, 0.11769983917474747, -0.33068668842315674, 0.34862077236175537, -0.21224692463874817, 1.0831042528152466, 0.6551652550697327, -2.001678705215454, -1.5079352855682373, -0.04564467445015907, -0.26121339201927185, -0.0862378403544426, 0.8836156725883484, -0.9504172801971436, -0.6808627247810364, -0.25592145323753357, -0.5075679421424866, 0.5325496792793274, 0.04352286085486412, 0.4984094500541687, 1.6185963153839111, 0.0945955142378807, 0.13831765949726105, -1.165326714515686, -0.6075171232223511, -1.697279691696167, -0.4106628894805908, -0.1300724297761917, 0.9347718358039856, 0.22212795913219452, -1.0294488668441772, 0.5044339299201965, 1.2183268070220947, -2.3605892658233643, 1.3263188600540161, -0.26498520374298096, -1.6850523948669434, 1.0317405462265015, -0.9177690148353577, -0.45250988006591797, 0.738039493560791, 0.22349782288074493, -0.8066297769546509, 1.0770504474639893, -0.7838215827941895, -1.105474591255188, -0.38531410694122314, 0.11394160240888596, 0.11965280771255493, 1.0017391443252563, -0.8539416193962097, -2.918555498123169, -0.4586145877838135, -0.1702231913805008, 0.3096526265144348, 0.9299790263175964, 0.3610647916793823, -0.07205110043287277, -0.9315451979637146, -0.2944583594799042, -0.8762962222099304, -1.984738826751709, 0.4567400813102722, 0.8311399817466736, -1.0538561344146729, 0.3560412526130676, 0.19485889375209808, -1.968355417251587, 1.9538633823394775, -1.3355239629745483, 0.38052159547805786, 0.5491486191749573, 0.9657244682312012, -1.116799235343933, 3.018639326095581, 0.4679467976093292, 0.21228951215744019, -0.3425237238407135, -0.08858879655599594, -0.8225910067558289, -0.26208850741386414, -1.089638352394104, 0.14953655004501343, 0.01889476552605629, -1.5918372869491577, 1.7510930299758911, 0.7483455538749695, 0.41790008544921875, -0.45446479320526123, -0.745185375213623, 1.177290916442871, 0.3938652575016022, -0.566357433795929, 1.5548852682113647, 0.660838782787323, -1.99017333984375, 0.32391366362571716, 1.2895972728729248, 0.06266575306653976, -0.631289541721344, 0.8653578758239746, 0.5580649971961975, 0.737388014793396, 2.1365256309509277, 0.9668537974357605, 0.5820310115814209, -1.6041053533554077, 0.5219933986663818, -0.6861048936843872, 0.7554521560668945, -0.7579916715621948, -1.1086217164993286, -0.8196736574172974, 0.14962051808834076, -2.566641330718994, -0.6692387461662292, 0.07815823704004288, -1.2342475652694702, 0.8944186568260193, 0.07854821532964706, 0.3004838526248932, 1.3160011768341064, 1.543423056602478, -0.40914538502693176, -0.5018157958984375, 0.8661978840827942, 0.2500692307949066, 0.214320570230484, -1.54639732837677, -1.1809991598129272, -0.5726876258850098, 1.176477074623108, -0.14363127946853638, -0.29362010955810547, -0.02638264372944832, -0.5871374011039734, 0.9835363626480103, 0.5486788749694824, -0.4376314878463745, -1.166745662689209, -0.5236143469810486, 2.1767046451568604, 1.3442161083221436, -0.6106865406036377, 0.395073801279068, -1.7723181247711182, -0.598720908164978, -0.564121663570404, 0.6683798432350159, -0.8413094878196716, -0.6608394980430603, -0.9725245237350464, -0.3684588074684143, 1.4020308256149292, -0.4543938934803009, -0.418367475271225, -1.7454512119293213, 0.08279518038034439, 1.060010552406311, 1.7322202920913696, -0.5482129454612732, 0.814721941947937, -0.2174953669309616, 0.39032113552093506, -0.12133465707302094, -0.39686939120292664, 0.44200554490089417, 1.0222952365875244, 2.4441170692443848, -1.4332901239395142, -1.0359513759613037, -1.7405762672424316, 0.24004249274730682, -0.5397158265113831, -0.8386391997337341, -1.888267993927002, -0.4813311696052551, 1.4531562328338623, 1.6475822925567627, 2.4142680168151855, 0.021402547135949135, 0.5948992967605591, -1.148258090019226, 0.7089881896972656, 2.5620718002319336, -0.7253651022911072, -0.9159984588623047, 1.4556646347045898, -1.765993595123291, -0.20700156688690186, -0.2243429273366928, -1.142783761024475, -0.7644413709640503, -0.802594006061554, 2.4832401275634766, 0.9870566725730896, 0.08767201006412506, 0.305883526802063, 1.4778141975402832, -0.23185423016548157, -0.1573946475982666, 0.12301595509052277, -0.33578696846961975, -0.9320157766342163, -0.8107036352157593, -0.4595608711242676, 0.8059881925582886, -1.3231561183929443, 0.36231642961502075, 1.3098955154418945, 1.6961168050765991, 0.2845389246940613, 0.7459363341331482, 0.29558804631233215, 0.5476778149604797, 1.406279444694519, -0.9452122449874878, 0.28475990891456604, 1.0353543758392334, 0.8383438587188721, 1.2600665092468262, 1.5457805395126343, 0.908054530620575, -1.9509168863296509, 1.6611781120300293, 0.4528137743473053, 0.2813314199447632, 0.4591970145702362, 0.3482428789138794, 0.10434219241142273, 1.1519651412963867, -2.0332252979278564, -0.7219793200492859, 0.6250271201133728, 0.4856582283973694, -0.9413262009620667, 0.12663854658603668, 0.4036489427089691, -0.5069281458854675, -1.6764355897903442, 1.8404595851898193, -0.5622902512550354, 1.164161205291748, -0.07541952282190323, 1.0885612964630127, -0.13819411396980286, -1.316380500793457, 0.6316757798194885, 0.06095746159553528, 1.3087313175201416, 1.942117691040039, 0.1440296322107315, -0.9659640192985535, -0.024359211325645447, -0.5337841510772705, -0.8261479735374451, -0.5271356105804443, 1.7121680974960327, -0.6222473978996277, -1.4784300327301025, 3.2950308322906494, 0.4113561511039734, 0.14742545783519745, 1.2172976732254028, -0.25003138184547424, 2.3201003074645996, 0.674255907535553, -2.023817300796509, 0.39293307065963745, -1.0231022834777832, -0.21204936504364014, -0.64350426197052, -0.3747928738594055, 1.9122377634048462, 1.465337872505188, -2.902761697769165, 0.7659220099449158, -1.4967777729034424, 0.016698554158210754, -1.7907838821411133, -0.44801783561706543, -1.5002204179763794, 0.9805433750152588, -0.8681796789169312, -2.283389091491699, -0.4215426743030548, -0.31425487995147705, -0.6813622117042542, -0.7324720025062561, 0.23723511397838593, -0.5441409349441528, 0.7274026870727539, -0.8443029522895813, -0.9536822438240051, 1.462041974067688, -1.177130937576294, -1.2562659978866577, 0.008769367821514606, 1.6736948490142822, 1.0048952102661133, 0.6202473640441895, -0.047763701528310776, -1.1857081651687622, 0.1506814956665039, 0.4511817693710327, -0.23299816250801086, -0.22966569662094116, 0.2507660388946533, -0.28750476241111755, 0.862855076789856, 0.2645248472690582, 1.118825078010559, -1.0203195810317993, -0.332974374294281], [-0.1490597128868103, -1.8586561679840088, 1.1174919605255127, 0.16605672240257263, 0.0682009607553482, -1.656190276145935, -0.9401819109916687, 1.8579567670822144, -0.381329745054245, -0.1761898547410965, -0.669365644454956, 1.9291236400604248, -0.1270214021205902, 1.0790538787841797, 1.2097440958023071, -0.19655799865722656, -0.4915168881416321, 0.14859463274478912, 0.9829950332641602, -0.0006026639603078365, -0.9506503939628601, 0.35483846068382263, 0.6475303769111633, 0.9556416869163513, -0.5105737447738647, 1.2707394361495972, -0.7997147440910339, -1.308840274810791, -0.16319595277309418, -1.4095336198806763, 1.026599407196045, -0.6877778172492981, -0.892422616481781, -0.19871248304843903, 1.8165545463562012, 1.484463095664978, -0.42016905546188354, 0.17121213674545288, -1.9484361410140991, 0.20295540988445282, 1.5228474140167236, -0.7688922882080078, -0.5057220458984375, 0.5854509472846985, 0.4389840364456177, -1.1559613943099976, -0.36406809091567993, 0.9287868142127991, 0.5478352904319763, 0.17655837535858154, -0.11613938212394714, -0.3778512179851532, -1.3875911235809326, -0.1192714050412178, 0.17698150873184204, -0.7421318888664246, 1.5691287517547607, 0.45371371507644653, 0.43674126267433167, -0.34710603952407837, 0.4537138044834137, 0.6347850561141968, 0.16043512523174286, -0.4884714186191559, -1.0896892547607422, -0.12700316309928894, 0.9376659393310547, 1.1076016426086426, -1.136527419090271, 0.6028876900672913, -1.327450156211853, -1.3915019035339355, 1.943959355354309, 0.12748275697231293, -1.6573212146759033, 0.019154366105794907, 0.16559426486492157, -0.04276558384299278, -2.749321699142456, 0.8305546641349792, -0.32118961215019226, -0.4783596694469452, 0.09429377317428589, -1.0403465032577515, 2.0452687740325928, -0.44134190678596497, -0.4032580256462097, -1.8050823211669922, -1.011668086051941, -0.6798489093780518, 0.28235068917274475, 0.1112423837184906, 0.7982536554336548, -0.13155773282051086, 0.15831364691257477, -1.0255801677703857, -0.3986469507217407, -2.154066801071167, -0.797408401966095, 1.246382474899292, 0.03551282361149788, -0.7002245187759399, 0.916471540927887, -0.8450468182563782, 1.632075548171997, 0.4103086590766907, -0.09773712605237961, 0.5792814493179321, -0.4283938407897949, -0.8126949667930603, -0.18792729079723358, 1.8512885570526123, 1.13705575466156, -0.8192044496536255, 0.43647974729537964, 0.5188709497451782, -0.466537207365036, -0.9284313917160034, -0.3613874614238739, -1.236587643623352, 0.7923476099967957, -0.17404049634933472, 0.7216038107872009, -0.7165554761886597, 0.41895386576652527, 0.13310952484607697, 0.4120769500732422, 1.1768791675567627, -0.6578488945960999, 1.2494583129882812, 0.48756641149520874, 0.6324948072433472, -1.5218136310577393, 0.9880849123001099, -0.7969722747802734, 0.513905942440033, 0.46413809061050415, 0.8217870593070984, 1.0289149284362793, -0.3648729920387268, 0.20843707025051117, 0.15758784115314484, -0.26489555835723877, -1.0081336498260498, -0.759150505065918, 0.17586325109004974, -0.227178156375885, -0.07329468429088593, -0.3261983394622803, -0.22033099830150604, -1.3665540218353271, 0.2754940092563629, -0.7245861887931824, -0.029476020485162735, 1.9522416591644287, 0.2918291687965393, -0.8190507888793945, -1.8245816230773926, 0.8547883033752441, 1.115667462348938, 0.5855161547660828, 0.5826172828674316, 1.480790138244629, 0.3113003075122833, 0.9084282517433167, -1.3301726579666138, 0.29404905438423157, -0.31674256920814514, 0.48135364055633545, -0.660637617111206, -0.38429394364356995, -0.14749382436275482, 0.9137622117996216, -0.701291024684906, -1.0254127979278564, 0.47239890694618225, 0.8619654178619385, -2.553450107574463, 2.3030805587768555, -0.09913863986730576, 0.09951217472553253, 1.013145923614502, 0.847564160823822, 1.4804729223251343, 0.29379504919052124, 0.4344412386417389, -0.6459262371063232, 0.04971630126237869, -0.16734567284584045, 0.48589468002319336, -2.052396774291992, -0.2201179414987564, -0.8243153095245361, 0.37202879786491394, -0.03509501367807388, -0.9352958798408508, -0.5840444564819336, 0.23341184854507446, -1.90090012550354, 0.007548103109002113, -1.2175689935684204, 2.5152270793914795, -2.6866884231567383, 0.8038814663887024, 1.1799198389053345, -1.3525712490081787, -0.9683298468589783, -1.5303527116775513, -0.22057369351387024, -1.4104816913604736, 0.41502565145492554, 0.7238547205924988, -0.4725736081600189, -0.3218693733215332, 0.9125041365623474, 0.7130412459373474, 0.4916045069694519, -0.5938835740089417, 1.006213903427124, 1.9100373983383179, 1.5898054838180542, -0.2396424114704132, 0.8832895159721375, -0.5770177245140076, -0.41123834252357483, 1.3903740644454956, 0.10853319615125656, 2.053239107131958, -1.0730515718460083, 0.4496242105960846, 1.2829316854476929, 2.804828643798828, -0.2739163935184479, 2.1532397270202637, -0.5420730113983154, 0.5421090722084045, -0.8816370368003845, -0.5026905536651611, 0.45407944917678833, 0.609897255897522, 0.1400166004896164, -1.2129894495010376, -1.7282967567443848, 0.9434232711791992, 0.44299566745758057, 0.9198275208473206, 0.5622333884239197, -0.46575069427490234, -0.47038036584854126, 0.2644899785518646, 0.4170137047767639, -0.40915247797966003, 0.4563791751861572, 1.5647650957107544, 0.1737017035484314, 0.734386682510376, 1.5089747905731201, 0.27040886878967285, 1.7996323108673096, 0.3713068962097168, 1.453612208366394, -2.066624641418457, -1.8679476976394653, 1.2822667360305786, -0.1553853452205658, -0.6195316314697266, -0.3052189350128174, -1.1131107807159424, 1.7061525583267212, 0.13359923660755157, 0.6929306387901306, 0.6607500314712524, 0.1588413417339325, 0.5865463614463806, -0.0813850611448288, -0.15702874958515167, -0.09656589478254318, -0.87242192029953, -0.5733373761177063, 0.7106536626815796, -0.6244146823883057, 0.2997298836708069, -0.07117438316345215, 0.42605161666870117, -1.075452208518982, -0.1749510020017624, -0.27693554759025574, 0.6541541814804077, 1.3129311800003052, 1.309132695198059, 0.613325834274292, -1.3634601831436157, -0.49986302852630615, -1.4567928314208984, 1.0045746564865112, -1.052183985710144, 0.6401357054710388, 0.3943093419075012, 1.3770060539245605, -1.1325960159301758, -0.8672013282775879, -1.1279096603393555, -0.5684561729431152, -0.194478377699852, 0.32625940442085266, -0.4920198321342468, -1.6798158884048462, 1.6716296672821045, -0.7139413952827454, -0.8379735350608826, 1.3470696210861206, -0.7180957198143005, -2.1914801597595215, 1.0073574781417847, -0.6427340507507324, -0.9518442153930664, 1.4908201694488525, 1.53727126121521, 0.31223946809768677, -0.9666092395782471, -0.5816665291786194, 0.5092225074768066, 0.5428870320320129, -0.5829372406005859, -0.5223190188407898, -0.46344417333602905, -1.0029752254486084, 0.1945589929819107, 0.21288424730300903, -0.42276644706726074, -0.9049413204193115, 0.05140253156423569, 1.3571702241897583, -1.832471251487732, -1.243531346321106, -0.04134871065616608, 0.036499522626399994, 0.04806461185216904, 0.3519563674926758, 0.23185808956623077, -0.7285827994346619, 1.1516356468200684, -0.8961548805236816, -0.34646889567375183, 1.5882744789123535, 0.3925822377204895, -1.0755656957626343, -1.353528618812561, 0.30736157298088074, 0.24523599445819855, -0.49196964502334595, 0.20678874850273132, -1.5205714702606201, 0.8872675895690918, 0.48179829120635986, 0.7987233400344849, -1.0991575717926025, -0.8127513527870178, 1.2484850883483887, -1.329413652420044, -1.3812367916107178, 0.6589555740356445, 0.6470354795455933, 0.3831574320793152, 0.15131458640098572, 0.10843916982412338, -0.7425366044044495, 0.5704756379127502, -0.007898297160863876, -0.9242464303970337, -0.06751709431409836, 0.730962872505188, -0.601410984992981, -0.38920679688453674, 0.37041351199150085, 1.0958430767059326, -1.090847134590149, 0.07689031213521957, 0.3749525845050812, -0.7555259466171265, -2.2058029174804688, 0.14405524730682373, 0.8111253976821899, 1.382538914680481, -1.3449935913085938, -0.13633766770362854, -1.2524430751800537, -0.6141108274459839, -1.5498137474060059, -1.1167834997177124, 0.3041343092918396, -0.5241934061050415, 0.7723972797393799, 0.33026671409606934, 0.4760391414165497, 1.1233028173446655, -0.30425116419792175, 0.02863391861319542, -1.0438555479049683, 0.9902511835098267, -0.14525777101516724, 2.5353896617889404, -0.581749677658081, 1.996833086013794, -0.6958228349685669, 0.3291977345943451, -0.9025718569755554, 0.21040219068527222, 0.9148740768432617, 1.2183901071548462, -0.24106073379516602, 0.0400145947933197, 0.5686478614807129, -0.16752205789089203, -0.501747727394104, 1.9258909225463867, -1.8837517499923706, 0.06903202086687088, 1.6648709774017334, -0.33813685178756714, -1.3412086963653564, -1.2931077480316162, 0.08612135797739029, -1.255515694618225, -0.8379018306732178, 1.1492749452590942, 0.4040859043598175, 0.4890916049480438, -1.064711332321167, 1.1471214294433594, 0.9270225167274475, 1.640364646911621, -0.26098859310150146, 0.6866262555122375, 0.03128809854388237, 0.9584148526191711, -0.025490297004580498, 0.629116415977478, -1.0082310438156128, -2.114400863647461, -0.19688008725643158, -0.3226472735404968, -1.7026528120040894, -1.0872949361801147, 2.1689295768737793, -0.35787269473075867, -0.7753546833992004, -1.0142422914505005, -0.4076094627380371, 0.7623980045318604, -1.1475951671600342, -1.2063136100769043, -0.8690548539161682, 1.6417230367660522, 0.42750248312950134, -0.31816449761390686, 1.4947768449783325, -0.2513861656188965, -1.069046139717102, -0.8107967376708984, 0.7608872056007385, 0.03246808424592018, 0.647055447101593, -0.020969130098819733, 0.5496211051940918, -1.1115890741348267, 0.21086432039737701, -0.39981162548065186, -1.1996824741363525, -0.49529311060905457, -0.5711971521377563, -2.141828775405884, 1.0798755884170532, -0.34923040866851807, -0.5864207744598389, 0.3427429497241974, -0.18147151172161102, -1.1612610816955566, 0.5244979858398438, 0.04396308586001396, -0.6770914793014526, 0.44663137197494507, 0.07110705226659775, -0.11105334758758545, 0.691267192363739, 0.540801465511322, -0.7177572846412659, 0.4276503026485443, 1.1389269828796387, 0.7966707944869995, 0.21539586782455444, 1.7400099039077759, 0.3326714336872101, 0.16779319941997528, 0.991828441619873, 0.9066576361656189, -1.3479710817337036, -0.4810522496700287, -0.5912385582923889, -0.5510361194610596, 0.8823685050010681, -0.5264515280723572, 0.1557466834783554, -0.9854690432548523, -0.8974989056587219, -1.658969759941101, 0.6990531086921692, -0.07799576967954636, -0.3980739414691925, -0.32929936051368713, -0.6881392002105713, -0.4215061366558075, -1.0730968713760376, 0.8279548287391663, -0.2914360761642456, -0.816891610622406, -0.23845776915550232, 0.2261064201593399, -0.04512631893157959, 0.03516248241066933, 0.9569766521453857, -0.10254033654928207, 0.6862061619758606, -0.37852758169174194, 1.3185818195343018, -0.8356073498725891, 1.4677844047546387, -1.0876637697219849, 0.32386478781700134, -0.3701532483100891, -0.23666000366210938, 0.1414858102798462, -0.010627384297549725, -0.52886962890625, -1.2688628435134888, 0.8499596118927002, 0.7389663457870483, -1.7580757141113281, 0.8478348255157471, 0.6599059104919434, 1.4890477657318115, -0.2292567640542984, 0.6033366918563843, -1.215480089187622, -0.01842508278787136, -0.49490678310394287, 0.6826725006103516, 0.5757191181182861, -0.0072030737064778805, -0.5280476212501526, -1.150062918663025, -2.8122429847717285, -0.0115195382386446, 0.8683193325996399, -1.1114839315414429, -0.11965154111385345, -0.9882071018218994, -0.1329536885023117, 0.9238688945770264, -0.6129474639892578, -0.034299835562705994, 0.5483832955360413, -2.0443081855773926, 0.06038788706064224, -0.6806656122207642, -0.22761084139347076, 1.636846899986267, -0.871508002281189, -1.0821211338043213, 0.6210558414459229, -1.2846981287002563, -0.4230925440788269, 1.0001646280288696, 1.0891413688659668, 0.1584259569644928, -1.085747241973877, -1.8780335187911987, -0.20530687272548676, -1.6603164672851562, 0.5580480694770813, -1.137641191482544, -1.219902515411377, -0.19189324975013733, -1.4932429790496826, -0.34882092475891113, -0.7532625198364258, 2.3679215908050537, -0.9909663200378418, -1.4528874158859253, -0.10372696816921234, 0.010818933136761189, -0.07457863539457321, 1.2355303764343262, 0.9276665449142456, 1.425804853439331, 0.9132518768310547, 0.9290194511413574, -2.2026524543762207, -1.071496844291687, -0.2842000126838684, -0.2159307301044464, -0.1730632185935974, 1.5922472476959229, 0.16040632128715515, -0.2278248518705368, 0.6664488911628723, 0.10205993056297302, 0.10121407359838486, -0.13262656331062317, 1.5186443328857422, -0.4614526033401489, -0.5598176121711731, -0.4964446425437927, 0.863918125629425, 0.08203939348459244, -0.9848108887672424, 0.7437776327133179, 0.5801036357879639, -0.23890377581119537, -0.2464212030172348, 0.9839736819267273, 0.014595801010727882, 0.20826870203018188, 0.42397379875183105, 1.8850561380386353, -2.689505100250244, 0.04530555382370949, 0.5775882005691528, 0.3220268785953522, 1.3866380453109741, -0.3621390461921692, -2.431058406829834, 1.1296390295028687, -2.0701215267181396, 0.2861807346343994, 1.1167988777160645, 0.7501028180122375, -2.151620864868164, 0.03946768492460251, -0.40506309270858765, -1.2867038249969482, 1.3931217193603516, -0.7886817455291748, -0.5984923243522644, -1.8941152095794678, -0.6319799423217773, 0.401752233505249, -0.6231632232666016, -1.097831130027771, -0.3245675265789032, -1.1556484699249268, 0.917278528213501, -0.8619261384010315, -0.8291252851486206, -0.3089345395565033, -1.5484012365341187, 0.3697412312030792, 1.5674768686294556, -1.1103148460388184, 0.31092369556427, 0.6848363280296326, -1.6429215669631958, -0.041849590837955475, -0.5739756226539612, 0.06522078812122345, 0.1772533506155014, -0.9724555611610413, -0.7578532099723816, -1.7665778398513794, 0.6871782541275024, -1.9520148038864136, -0.4606621563434601, 0.3658280372619629, 0.5469083189964294, 1.0229196548461914, 0.3119688928127289, -1.2703962326049805, 0.7628727555274963, 2.2805910110473633, 0.7818471193313599, -1.023945689201355, -1.236244797706604, 0.04726475849747658, 0.9260094165802002, 0.2383953034877777, 0.9087556600570679, -0.4333433508872986, 0.06719613075256348, 0.26648664474487305, -0.8035610914230347, -0.18247513473033905, -0.31043541431427, 0.4621593654155731, 0.9388707876205444, 0.15213364362716675, 1.6859382390975952, -0.3866039514541626, -1.787620186805725, 0.9911400675773621, -0.12390229851007462, -0.6733344793319702, 1.060853123664856, -0.9897949695587158, -2.305551052093506, 0.14272955060005188, -0.3403274714946747, 1.1405723094940186, -0.8491125106811523, -1.6796751022338867, -0.2228303700685501, 2.093724250793457, -2.034949779510498, 0.19959186017513275, -0.03617442771792412, -0.5198418498039246, 0.23664745688438416, 0.19631527364253998, -1.1423931121826172, 0.16135206818580627, -0.2276764065027237, -0.6056271195411682, 0.48520779609680176, 0.8396152257919312, 0.5364069938659668, -0.2911704480648041, -0.7611232995986938, -0.47775858640670776, 1.4140214920043945, -0.4058310389518738, -0.2715601325035095, -1.2512941360473633, 1.1190128326416016, -0.25711336731910706, -0.642266571521759, 0.5138425230979919, 1.222091555595398, 0.24792274832725525, 0.604063868522644, -0.09182697534561157, -1.000649094581604, 0.014029202051460743, -0.9294717907905579, 1.8455455303192139, -0.3397689163684845, 0.05602462962269783, -0.2055654674768448, -0.7248778939247131, 0.7750592231750488, -0.14827433228492737, -0.5278931856155396, -3.3553202152252197, -0.4038706123828888, 0.8169012665748596, -0.36975833773612976, 1.7528817653656006, -0.3940035402774811, -0.6359975337982178, -1.5423108339309692, -0.5349084734916687, -0.4739525020122528, 1.078289270401001, 0.13212241232395172, 0.7615334987640381, 0.015597858466207981, -1.3349127769470215, -0.5738124847412109, 0.1067870557308197, -0.4694536626338959, 0.48889702558517456, 2.0184483528137207, -1.5336061716079712, 0.4115590453147888, -1.5150946378707886, 0.5543571710586548, -0.46352264285087585, 0.33566156029701233, 1.177075743675232, -0.29700976610183716, 0.38435298204421997, 1.7705022096633911, -0.0680670514702797, -0.6303431391716003, 1.6585874557495117, 1.8387274742126465, -0.37714818120002747, -0.4880915582180023, 1.031202793121338, -1.282379150390625, -0.8262373208999634, 0.37295907735824585, -0.01900455169379711, -1.064797282218933, -0.4853784143924713, -0.21469298005104065, 0.2386208474636078, 1.7529220581054688, -0.6419556140899658, -1.9832277297973633, 1.0275830030441284, 0.08995603770017624, 1.093155026435852, 0.36630845069885254, -1.2552692890167236, 1.2600430250167847, -1.4445257186889648, 0.9958320260047913, -0.42393118143081665, -0.6721879839897156, 1.4345581531524658, -0.26473790407180786, 0.8432826995849609, -0.130680650472641, 0.4707753360271454, -1.9001314640045166, -0.36306431889533997, 1.41154944896698, -0.4501090943813324, -0.6427040696144104, 1.1049262285232544, -0.6698331236839294, -0.8616622686386108, -0.3245813846588135, -1.222544550895691, 0.005942700896412134, -1.034271240234375, -1.1403590440750122, 1.517492413520813, -0.5253827571868896, 0.13883444666862488, 0.6990919709205627, 0.8574451208114624, 1.7712986469268799, -1.1715805530548096, -0.028157230466604233, 1.2666174173355103, 1.4257566928863525, -0.3283775746822357, -0.04701399430632591, -0.08596301823854446, -0.7920379638671875, -0.8924010396003723, 0.32422739267349243, 0.8820391297340393, 0.37607911229133606, 0.48033836483955383, -0.771702766418457, -1.3412736654281616, -0.5923713445663452, 0.16201151907444, -0.6062762141227722, -0.25927430391311646, -0.7048670649528503, -1.084190011024475, -2.4473512172698975, 1.0371575355529785, -0.767894983291626, -0.9230980277061462, 0.17079688608646393, 2.777747392654419, 0.7978845238685608, 0.3597849905490875, -1.1687456369400024, -0.8209978342056274, 1.120278239250183, 0.9467072486877441, -0.03854239732027054, -1.0919857025146484, -0.8866007328033447, 1.6701350212097168, -0.49799856543540955, 0.0410037524998188, 0.7900282740592957, 1.0023547410964966, 1.74382483959198, -0.2891405522823334, -1.638343095779419, 1.6184451580047607, -1.701627254486084, 1.5510380268096924, 1.5997544527053833, 1.1790754795074463, 0.26948362588882446, 0.8404425978660583, -1.4771782159805298, 0.744521975517273, 0.9281685948371887, 0.20801177620887756, 1.618377923965454, 1.473207950592041, -0.7146143317222595, 0.2846948504447937, 0.3858794867992401, -0.07117179036140442, 2.134976863861084, 1.552258849143982, 0.6559233665466309, 1.831790566444397, -0.8818512558937073, -0.33425983786582947, -0.2922634780406952, 0.28895634412765503, -0.19907166063785553, 0.41181400418281555, 0.09720773249864578, 1.8873883485794067, 2.255340337753296, -0.7973784804344177, -1.4595351219177246, 0.4446094334125519, -0.8895827531814575, -0.9426549077033997, 0.12680837512016296, 0.7109485864639282, -0.9484967589378357, 2.252645254135132, 0.3878033459186554, -0.3978560268878937, 0.09115242958068848, -0.9914332628250122, -1.6044069528579712, 1.1842865943908691, -0.29548683762550354, -0.5838760733604431, 1.3062489032745361, 0.20438364148139954, 1.5333540439605713, -0.7432171702384949, 0.1664825826883316, -0.8585850596427917, -0.033529557287693024, 0.08391430228948593, 1.1164460182189941, 0.0014252383261919022, -0.7818036675453186, -0.8281126618385315, -0.6585381031036377, -0.7546303868293762, -0.8017441034317017, 0.21012185513973236, 0.1440524160861969, 0.6392687559127808, -1.4247857332229614, -0.7166606783866882, -0.22986873984336853, -1.5027506351470947, 1.6980137825012207, 0.002830818295478821, 2.5774614810943604, -0.5583099722862244, 0.12915022671222687, -0.7896377444267273, 0.05607856065034866, -0.43997013568878174, 0.6421429514884949, 0.7912790179252625, -0.20563767850399017, 0.3739945888519287, -0.2544792890548706, 1.0662356615066528, 1.349709153175354, -1.1416313648223877, 0.31448233127593994, -1.005647897720337, -1.139426827430725, -0.5682728290557861, 1.624554991722107, -0.8537566065788269, -0.4292377829551697, -0.12918917834758759, -1.0252701044082642, -1.6023272275924683, 2.2955729961395264, -0.5277501344680786, -0.5074373483657837, 1.4827340841293335, -0.7881249785423279, 0.9256337285041809, -1.2164700031280518, 0.5802052617073059, -0.09175511449575424, -1.9610276222229004, -1.3251405954360962, -0.7841886878013611, -1.828425407409668, 0.14438489079475403, -1.5535056591033936, 1.9570260047912598, 0.6555947065353394, 1.2544687986373901, -0.12684468924999237, 0.9641188979148865, 2.1360292434692383, -0.8683791160583496, -0.7067850828170776, -0.144223153591156, -0.9541358947753906, -0.9595806002616882, -0.9612470865249634, 2.3797450065612793, -3.343461751937866, 0.17357143759727478, -1.0052469968795776, -0.06678014993667603, 0.46059292554855347, 1.1193088293075562, 1.0158226490020752, -0.27559101581573486, 0.28940802812576294, 0.2223520576953888, -1.0870652198791504, 0.6257542967796326, -1.5857276916503906, -2.2556824684143066, -0.31403717398643494, -0.35369887948036194, 0.4278004765510559, -1.5248233079910278, 0.6687201261520386, -0.5154982209205627, 1.4172992706298828, 0.5292167067527771, -0.1422388255596161, 0.8179959654808044, -0.9608676433563232], [0.48828014731407166, -1.7251744270324707, -0.47152161598205566, -0.3874554932117462, -1.8667281866073608, 0.43892011046409607, -0.7435158491134644, 0.8997418880462646, -0.7310288548469543, -1.39405357837677, 0.1003611832857132, -1.6721147298812866, 0.3092295527458191, 1.23455011844635, -0.38889622688293457, -0.4586930572986603, 0.5731009244918823, -0.0993911474943161, 0.8105039596557617, 0.2820092737674713, 1.4493893384933472, 0.14939486980438232, 0.038940176367759705, 1.9355841875076294, 0.31148332357406616, 0.3245329260826111, 1.1406341791152954, -0.36500880122184753, -0.3168414831161499, -0.854584813117981, 1.0222598314285278, 0.8358232378959656, 0.20818273723125458, -1.5406461954116821, -0.05153147503733635, -0.5129349231719971, -0.24228575825691223, -1.2552452087402344, 1.3783067464828491, 0.5379518866539001, 0.6766548156738281, 0.5498889684677124, 1.3903580904006958, -0.9525877237319946, 0.6281948089599609, 0.21864184737205505, -0.6829277276992798, -0.1732022613286972, -1.4946918487548828, 0.404848575592041, 1.420031189918518, 0.32133421301841736, -0.31125664710998535, 0.27567145228385925, -1.5981873273849487, 0.40047845244407654, 0.48118066787719727, -0.5593830347061157, -0.13342973589897156, -0.7694316506385803, -0.6042932271957397, -0.9869934916496277, 0.3043927550315857, 0.18132995069026947, 1.2840625047683716, 1.3555872440338135, -0.5807747840881348, -0.21488022804260254, 0.9376509189605713, 0.16491369903087616, 1.3096972703933716, -0.15339843928813934, 0.14004316926002502, -0.822338342666626, 0.5491107106208801, -1.4051765203475952, -0.19132345914840698, 0.52281254529953, 0.2751270830631256, 0.10819092392921448, 0.5795087218284607, -0.4202962815761566, 1.3105921745300293, -3.0853683948516846, -0.2585829496383667, 0.7753122448921204, 0.9048542976379395, -1.925026535987854, 0.3737829029560089, 0.9886646866798401, 0.20670929551124573, -2.09513783454895, 0.9229584336280823, -0.7188915610313416, 0.22616475820541382, -1.5692747831344604, 0.35288047790527344, -0.5452877879142761, 0.1306958943605423, -2.0652356147766113, 0.551507294178009, -0.1556028425693512, -0.7417003512382507, -0.2688356339931488, 0.771734893321991, -1.0584701299667358, -1.0913969278335571, 2.4733524322509766, -1.2114386558532715, 0.21183420717716217, -0.45534855127334595, -0.15096735954284668, 0.010996066965162754, -0.9289580583572388, 0.4972202181816101, -0.7050486207008362, -1.4228860139846802, 0.987934947013855, -0.6958942413330078, 0.24497176706790924, -0.2272813767194748, -1.0205068588256836, 0.5227387547492981, 1.1881756782531738, -0.4306752681732178, -0.03677522763609886, -0.34799525141716003, -0.41688430309295654, -0.742805004119873, 0.6925480365753174, -1.984822154045105, -0.7888027429580688, 1.0704646110534668, 0.2059958428144455, 1.070965051651001, -2.049196481704712, -0.3197290897369385, 0.16200187802314758, 0.5066230297088623, -0.5071183443069458, 0.941562831401825, -0.36264851689338684, 0.6803344488143921, 0.4367483854293823, -0.33599525690078735, 0.019046686589717865, -1.654158353805542, 0.1734219789505005, 1.074397325515747, 0.7681074738502502, 0.7433151602745056, -0.6771575808525085, 0.12203897535800934, -0.27602916955947876, -0.8856185674667358, 0.025223376229405403, 0.6845980882644653, 0.44199496507644653, 0.1179303303360939, -0.4341346025466919, -0.6877915263175964, -0.09323923289775848, -0.44128701090812683, 1.4361698627471924, -0.33505013585090637, -0.0926426500082016, 0.05383845046162605, -0.9881240725517273, 0.8430878520011902, -0.3314935266971588, 0.4421662390232086, -0.8669843077659607, -0.2804299592971802, 1.040757417678833, -0.2986126244068146, 0.14458073675632477, -0.5534358024597168, 0.43177366256713867, 0.5507470965385437, 0.37219691276550293, 1.0071954727172852, -0.8644795417785645, 0.2349073737859726, -0.009038069285452366, -0.7972583174705505, -1.653432011604309, 0.7190017700195312, 1.9211125373840332, 0.8401763439178467, -1.1039105653762817, 0.5687517523765564, 0.28319138288497925, 0.47009408473968506, -0.39505454897880554, -1.3081247806549072, 0.8422219157218933, 0.6010158658027649, 0.06356905400753021, -0.2640719711780548, -0.3980361521244049, 0.7120364904403687, 0.9774417877197266, 0.18688832223415375, 0.13595004379749298, 0.6867049336433411, -0.7479743957519531, 0.8485482335090637, -1.0771006345748901, 0.5421176552772522, -0.8732335567474365, 1.973846197128296, -1.4162724018096924, -0.05524077266454697, -1.0353463888168335, 0.3124045133590698, 0.6458232402801514, 0.390277624130249, -1.2828960418701172, -1.7951090335845947, 0.5332813262939453, -0.6651812791824341, 0.300680547952652, 0.1344984620809555, -0.9117796421051025, 1.1622247695922852, 2.307011604309082, -1.147175908088684, 0.2822931408882141, -1.1284259557724, 0.775976836681366, -0.14043977856636047, -0.156050443649292, -0.33086881041526794, 0.8637571334838867, 1.0419085025787354, 1.2072359323501587, -0.7137662172317505, 1.1648199558258057, -0.14102713763713837, -1.2637138366699219, 1.4260412454605103, -0.6144415140151978, -0.8247369527816772, 0.5966026186943054, -1.1057662963867188, 0.7014431357383728, 1.9433650970458984, 1.364061951637268, -1.2562512159347534, 0.6479036808013916, 0.9365692734718323, 1.2066268920898438, 0.5358726382255554, -0.9380427002906799, -0.30106645822525024, 1.782291054725647, -0.5501523017883301, 2.239044666290283, 0.23119354248046875, -1.2520986795425415, 0.07590138912200928, -0.4218621253967285, 0.8288135528564453, 0.09239379316568375, 0.5994372963905334, 0.15509870648384094, -0.30028852820396423, -2.13830304145813, 0.3270692527294159, 0.26991042494773865, -1.0267727375030518, -1.2923378944396973, -0.3719582259654999, -0.23571813106536865, 1.0382705926895142, -0.7773759365081787, -0.054530709981918335, -0.0014556144597008824, 0.3849412202835083, -0.8224436044692993, -0.40384185314178467, -0.7518308758735657, 0.06974857300519943, -1.6025217771530151, -0.23695522546768188, -1.1069817543029785, 0.48186296224594116, -0.4946155846118927, 0.2220684438943863, 0.22210566699504852, 0.48125773668289185, -0.5959630012512207, 1.620762825012207, -0.05429338291287422, 0.15224036574363708, 1.5516977310180664, 1.041995644569397, 0.016097938641905785, 0.2027217298746109, 1.0826970338821411, 1.4522020816802979, -0.6597927808761597, -0.9118801355361938, -1.0602633953094482, 0.5290529727935791, -0.24529337882995605, 2.2398083209991455, -0.13084010779857635, 0.7327420115470886, -0.9035962820053101, 0.13642802834510803, -0.8770579099655151, 0.7716163992881775, -0.4434279203414917, -0.39664435386657715, 1.6217790842056274, -0.8660105466842651, 0.2453906536102295, 0.5096230506896973, -0.3491687774658203, 0.6511859893798828, -0.15617190301418304, -2.196737051010132, -0.5405089259147644, 0.23668619990348816, 0.22842980921268463, 1.0995001792907715, -1.5442142486572266, -0.7183454632759094, -0.530003011226654, 1.079041838645935, -1.151259422302246, 0.3301880657672882, -0.4175642728805542, -0.5196179747581482, 1.2248106002807617, 0.1689618080854416, 0.3130771517753601, 1.4130014181137085, -0.4988650381565094, 1.5699429512023926, -1.4821202754974365, -0.9645934104919434, -0.6290159821510315, 0.41623806953430176, -0.04673219472169876, -1.105334758758545, -1.1318544149398804, 1.9944871664047241, -0.35915407538414, -0.6154635548591614, 0.09846854209899902, 0.6437296271324158, 1.052350640296936, 0.58729088306427, -0.6307281255722046, 0.4825994074344635, 0.3922867476940155, 0.7904882431030273, 2.184995174407959, -0.6104281544685364, -0.7432883977890015, -1.474727749824524, 0.6879689693450928, -0.9586265087127686, 1.5306105613708496, 1.038688063621521, 0.14176368713378906, 0.5343916416168213, -0.3951869606971741, 0.6515867710113525, 0.3885933756828308, 2.0122995376586914, 2.116105794906616, 0.7154598236083984, -0.8135035634040833, 0.505617082118988, -0.866940975189209, -0.22406105697155, -0.9415554404258728, 0.5885766744613647, 0.5440260171890259, 0.31030505895614624, 0.8072710037231445, -0.7879199981689453, -2.497081756591797, -0.3602748513221741, -0.2602004408836365, 1.447593092918396, -0.07476164400577545, 0.5863965749740601, 0.31493037939071655, 1.032440185546875, 1.5104368925094604, 0.1844499558210373, 0.22399258613586426, -0.9943366646766663, 1.3773747682571411, 1.5011932849884033, 0.22055691480636597, 0.04401491582393646, 0.45886850357055664, 0.3852681815624237, 0.3992519974708557, 0.9112647175788879, -0.8170014023780823, 0.5157201886177063, 0.6557636260986328, -0.536821186542511, 0.8124345541000366, 0.25976628065109253, 0.04298461228609085, -1.6845707893371582, 0.10298556834459305, -0.1263565868139267, 0.24582725763320923, -1.0194833278656006, 1.4502367973327637, 0.7108227610588074, -0.14557962119579315, -0.3225759267807007, 1.0686616897583008, 0.651217520236969, -1.1626288890838623, -0.23225921392440796, 0.7649322152137756, -0.7070333361625671, 0.1408085823059082, -0.8277832865715027, -0.7286086678504944, 0.5446380972862244, 1.1172425746917725, 0.14367520809173584, -1.6498361825942993, -0.274334579706192, -0.366355299949646, -0.8233409523963928, -0.19372305274009705, 0.558535635471344, -0.7229296565055847, 0.6562950611114502, 0.19678184390068054, -0.5672739148139954, 0.6318274736404419, 1.1609050035476685, 1.2253755331039429, 0.10587365180253983, 1.8615834712982178, 0.2683699429035187, -0.9454350471496582, -0.19949299097061157, -0.036293208599090576, -0.24886339902877808, -0.22574281692504883, 1.431593418121338, -1.1382651329040527, 0.24149905145168304, 1.0292339324951172, 1.0957518815994263, 0.7713184356689453, -1.767900824546814, -0.8962311148643494, -1.1791062355041504, -0.9147336483001709, 1.6524897813796997, -1.323426365852356, -1.4778938293457031, 1.276220440864563, -0.4934476613998413, -2.7690589427948, 0.14146602153778076, 1.9932862520217896, -0.05983251333236694, 2.854645013809204, -1.4522582292556763, -0.26874515414237976, 0.5134293437004089, 1.1618618965148926, 0.04465601593255997, -0.8762730360031128, 1.9851032495498657, 0.5345166921615601, -2.219326972961426, -0.02945823222398758, 0.3307442367076874, 2.1830358505249023, -0.8192160725593567, 0.2960636615753174, 0.8025681376457214, 1.22371506690979, -0.2831255793571472, 0.41871029138565063, -0.0036761045921593904, -1.4283725023269653, 0.5810704827308655, -0.6949387192726135, 2.2242839336395264, -0.33769911527633667, -1.8812633752822876, 0.2747204899787903, -1.6740084886550903, -0.909451961517334, 0.5471175909042358, -0.6360993385314941, 0.20909036695957184, 0.9936125874519348, -0.47401508688926697, 1.3478552103042603, -0.005924080964177847, -1.1249572038650513, 0.007158844731748104, -1.295102596282959, -0.18248246610164642, 1.664400339126587, 0.2186374068260193, 0.34992724657058716, -0.1858023852109909, 0.7843574285507202, 0.4927263855934143, 1.014769196510315, -0.9608523845672607, -2.081965684890747, 1.2849582433700562, -0.3853290379047394, -0.6336069703102112, 0.22176247835159302, -1.3127334117889404, 0.8552957773208618, -0.4908226430416107, -0.42797183990478516, -1.3360360860824585, -0.6470077037811279, -0.9571340084075928, -0.2344128042459488, 0.7786442041397095, 0.13281646370887756, -0.3664036989212036, -0.16686996817588806, -0.49256595969200134, 0.25038954615592957, -0.6792787909507751, -0.8331416249275208, -0.4228869378566742, -1.0463707447052002, -0.001357112661935389, -0.9115270376205444, -0.19285070896148682, -0.9817056059837341, -0.2938007116317749, 0.9589623212814331, 0.27883175015449524, -0.39003992080688477, 1.2337652444839478, 1.3730885982513428, -0.46874919533729553, -1.6584793329238892, -0.15363608300685883, -1.3634392023086548, 0.455654114484787, 0.6052402853965759, -0.48577964305877686, -0.7069569826126099, -1.2911649942398071, 0.9019193649291992, 1.0178048610687256, -0.7416828274726868, -0.9576212763786316, -0.544184148311615, 1.5954104661941528, -0.13986994326114655, 0.6794116497039795, -0.5551261305809021, -0.13339538872241974, -0.6961768865585327, -1.407939076423645, -0.02524992823600769, -0.7706032395362854, -0.12047787755727768, -0.3754049241542816, -0.5971585512161255, -0.363196462392807, -1.2446479797363281, 0.26330509781837463, -0.3433613181114197, -0.5738381743431091, 0.3015989363193512, -2.3584189414978027, 1.64321768283844, 0.8571092486381531, -2.3957443237304688, -0.015003609471023083, -0.6147604584693909, -0.07431702315807343, 0.627228856086731, -0.49095654487609863, 1.3954190015792847, -0.3384387791156769, -0.41694188117980957, 0.7359437942504883, -2.632251024246216, 0.5896410942077637, -0.3215428292751312, -1.1303263902664185, 1.5449496507644653, 0.23312680423259735, 0.1255790740251541, -0.10143140703439713, -1.3298473358154297, -0.5907081365585327, -0.6155561804771423, -0.34391316771507263, -0.149454727768898, 1.5113376379013062, -0.5280691981315613, 0.24759452044963837, -0.37658408284187317, 0.5438826680183411, -0.73663729429245, -0.019486263394355774, 0.4682672321796417, -0.5233036279678345, -0.9893697500228882, -1.0290215015411377, 0.3394729793071747, -0.1869901418685913, 0.2798450291156769, 0.34114840626716614, -0.12290313094854355, 0.5176019072532654, -1.4763389825820923, -1.4946681261062622, 0.1302422285079956, 0.29733943939208984, -0.06758031249046326, -0.5455735921859741, -0.8673118948936462, -1.1116727590560913, -0.16250622272491455, 0.3616512715816498, 0.7059264779090881, 0.4030509293079376, 1.807075023651123, -0.055424176156520844, -1.8717619180679321, -0.8821805715560913, 0.09609333425760269, -0.5865732431411743, -1.9013341665267944, 0.11232855170965195, 0.4135506749153137, -0.4161641001701355, -1.0988054275512695, -1.1003649234771729, -0.7388997673988342, -0.9058523178100586, -0.5645831823348999, -1.093401312828064, -1.609958291053772, 2.0919582843780518, -3.3059942722320557, -1.2397270202636719, 1.2343311309814453, 0.14429596066474915, -1.063709020614624, -0.24196843802928925, 1.2241188287734985, -0.6344637870788574, 1.4093689918518066, -0.9579204320907593, 0.21091261506080627, 1.003615140914917, -0.5229167938232422, -0.637846827507019, 1.5821280479431152, 1.8459972143173218, 0.5896755456924438, -1.1708462238311768, 0.008232192136347294, -1.0100067853927612, 1.5956175327301025, 0.8693138360977173, -0.05488995462656021, 0.02694748528301716, 2.009617805480957, -1.0498480796813965, 0.3782851994037628, 0.710332453250885, -2.808130979537964, 0.41718238592147827, 1.2650914192199707, -0.3581353425979614, -0.42917248606681824, -0.9672717452049255, -1.2330397367477417, -0.47925201058387756, 1.100507140159607, 0.09306994825601578, 0.3583976924419403, 1.23171865940094, -0.5744093060493469, -0.45256921648979187, 0.3939484655857086, 0.6941323280334473, 1.3647152185440063, 0.025900429114699364, -0.09645440429449081, -0.25662264227867126, -1.1547796726226807, -2.351468086242676, 0.05182314291596413, -1.1877250671386719, 1.1464335918426514, -1.1323795318603516, -0.13210086524486542, -1.1800837516784668, -0.5735227465629578, -1.3361209630966187, 0.9715785980224609, -0.2891140580177307, -0.5576649308204651, -0.22304533421993256, 0.3929591178894043, -0.24828873574733734, 0.630367636680603, -0.27630066871643066, -1.6393625736236572, -1.7859070301055908, 2.1132726669311523, 0.5415652990341187, -1.217158317565918, -1.3582571744918823, -0.6926922798156738, -0.5428130626678467, -1.0788453817367554, 0.08679413795471191, 0.5526989102363586, -1.7832615375518799, 0.01051820907741785, -1.1678946018218994, 0.17989739775657654, 1.4504731893539429, 0.32004988193511963, 0.7273625135421753, 0.04962089657783508, 0.7903622984886169, 0.1254286915063858, 1.3497110605239868, -0.028632551431655884, 0.8979536294937134, 0.9220436215400696, -0.45846256613731384, 0.3707704246044159, -0.8689833283424377, -0.0880257785320282, -0.7082355618476868, 0.6803939342498779, -0.5197607278823853, 0.6826723217964172, -0.34485894441604614, 0.7546405792236328, -0.5809178948402405, 0.7886708974838257, 0.4931046664714813, 0.7315136790275574, -1.2035369873046875, -2.3151848316192627, 1.04408597946167, 1.850719690322876, -1.8800371885299683, 1.282504677772522, -0.28304192423820496, 2.423959493637085, -0.23449571430683136, 0.09495064616203308, -1.2312294244766235, 0.13940787315368652, 0.5811753273010254, -0.1619768887758255, 1.307798981666565, 0.2520873248577118, -1.3703166246414185, 0.18639631569385529, 0.26369428634643555, -1.6794469356536865, 0.6713561415672302, -0.7237269282341003, -1.201825499534607, 0.15179336071014404, 0.3523658215999603, 0.9789595007896423, -0.9053184390068054, -0.1172366738319397, 0.9256603121757507, -0.2628914415836334, 1.0582783222198486, -0.9919790029525757, -0.7649043798446655, -1.0675281286239624, -0.2892118990421295, 0.13398778438568115, -1.717947244644165, 0.2312503159046173, -0.7961810827255249, 0.28233781456947327, -0.11459980905056, -2.43100643157959, -0.43246975541114807, 0.2993433475494385, -1.21413254737854, -0.07627584785223007, 0.9434689283370972, -0.9479102492332458, 0.011685543693602085, 2.2326090335845947, 0.06613661348819733, -1.081125020980835, 0.11601859331130981, -0.15696188807487488, 0.5678181052207947, 1.10040283203125, -0.3897082209587097, -0.3346664607524872, -0.8100572824478149, 0.5087540745735168, 0.7144365310668945, -0.15945789217948914, 0.7250889539718628, 0.4641191363334656, 0.8383537530899048, -0.6255966424942017, -0.5186337828636169, -0.6029853224754333, 0.5811318159103394, 1.2334003448486328, -1.4591280221939087, 0.19295580685138702, 1.0203680992126465, -0.6354720592498779, 1.1629680395126343, -0.5365176796913147, 1.171180248260498, -0.5163241624832153, 1.2710363864898682, -0.6686305403709412, 1.3440684080123901, 0.4932032823562622, 0.9135844111442566, 0.8365015387535095, -0.22933489084243774, 1.0060354471206665, 1.2342586517333984, 0.05623069033026695, -0.7403730154037476, 0.2238183468580246, -0.10797082632780075, 0.22798314690589905, -0.40181365609169006, -0.33581069111824036, -0.30227938294410706, 0.8497521281242371, -2.104762315750122, 1.9122604131698608, -0.7788981199264526, -0.6894500851631165, -0.8304598927497864, -0.6101628541946411, 0.12291219830513, -0.6643174290657043, -0.2152208834886551, 0.4675993323326111, 0.1528605818748474, -1.0752229690551758, -1.322901964187622, 0.4063848853111267, 0.8203251361846924, 1.023274540901184, -1.1645011901855469, 0.9542934894561768, 0.04713668301701546, -0.001756931422278285, -0.5609037280082703, -1.4534237384796143, -0.3896813690662384, 0.5070339441299438, 0.3621206283569336, -0.6719278693199158, 0.31353554129600525, 0.16078032553195953, 1.338101863861084, 0.04405735060572624, 1.1859270334243774, 1.846622109413147, 0.16258159279823303, 0.8788431286811829, -0.32749420404434204, -0.8751060962677002, 0.04657730087637901, 0.728706955909729, 0.4974606931209564, -0.6671632528305054, -1.34109628200531, 1.5346444845199585, -0.8455398678779602, -1.9751992225646973, -1.986598014831543, -0.5148656368255615, -0.016177212819457054, 0.9505285024642944, 1.0669894218444824, 1.5756545066833496, 1.016453742980957, -0.8817815780639648, -0.8869444131851196, 0.6929641962051392, -1.2363361120224, 0.23210768401622772, -0.6135659217834473, 0.808944046497345, 0.8628581166267395, -1.3849684000015259, -0.07539284229278564, -0.5094817876815796, 1.1430281400680542, -0.47709351778030396, 0.5203704833984375, -1.8042645454406738, -0.20480218529701233, -0.10478266328573227, 0.8683657646179199, 1.0093141794204712, -0.7779766321182251, -0.8193195462226868, 1.8946516513824463, 1.586673378944397, -0.3498552739620209, 0.449668824672699, -0.415596604347229, -1.218324065208435, 2.8842172622680664, -0.7984008193016052, 1.8244099617004395, 0.5323656797409058, 0.16657868027687073, 0.5653918981552124, 0.16309109330177307, 0.960361659526825, 1.0849708318710327, -1.180776596069336, 0.27364253997802734, 1.4115264415740967, -0.6751285195350647, -0.1282711923122406, -1.427305817604065, -0.01812131144106388, 1.3253402709960938, 1.2524412870407104, 0.7090986967086792, 1.1908745765686035, -0.11133550852537155, 0.3678920865058899, -0.788890540599823, 0.017466392368078232, -0.21322356164455414, -0.9600929021835327, -1.0220173597335815, 0.1520029902458191, -1.343598484992981, -1.0737426280975342, 0.33812952041625977, -0.24625247716903687, -0.7944692969322205, -0.43948620557785034, 1.4321396350860596, 1.2617491483688354, -0.851965606212616, -0.8069713711738586, 1.104645013809204, 1.1360515356063843, -0.7284687757492065, -0.09677598625421524, 0.8239031434059143, -0.5928047299385071, 0.563028872013092, -2.1900885105133057, -0.37016499042510986, 0.9256588816642761, 0.29249805212020874, 0.09115725010633469, -0.018989460542798042, -0.35228288173675537, 0.1482713371515274, -0.4805286228656769, -1.0416144132614136, 0.3065716326236725, 0.10609006136655807, -0.4957418441772461, 1.4491878747940063, -0.4876460134983063, 1.8966305255889893, -0.1494927704334259, 0.7067292928695679, 0.4645659029483795, -1.0535862445831299, -0.8452308177947998, 0.8285483121871948, -0.9838913679122925, 0.490469753742218, 1.1941548585891724, 0.033235251903533936, 0.4293164908885956, -1.9656116962432861, 0.49651026725769043, -0.6182351112365723, -0.36531928181648254, 2.3391268253326416, 0.6007553935050964, -0.3243229389190674, 1.4817497730255127, 0.4792451560497284, 0.620526134967804, 1.4619091749191284, 1.4998822212219238, 0.05490294471383095, -0.46058619022369385], [0.7946382761001587, 0.2758893668651581, 0.1618012934923172, -0.5345713496208191, 1.3881138563156128, 0.92686527967453, -0.277852326631546, -0.7663048505783081, -0.6729860901832581, 2.2426607608795166, -1.6991956233978271, 1.088748574256897, -1.1237354278564453, -0.006679362617433071, -1.0635029077529907, 1.2226496934890747, -1.1426012516021729, 1.0970399379730225, -0.3435194194316864, 1.7280856370925903, 0.3708046078681946, 1.1099576950073242, 2.3305816650390625, 1.5129505395889282, -0.6663091778755188, 0.22752150893211365, 1.3025546073913574, 0.3360617756843567, 1.0911108255386353, -0.774841845035553, 0.7751169800758362, 2.069763660430908, -0.5675780773162842, 1.4607305526733398, -0.09647269546985626, 2.889033317565918, -0.4975403845310211, 1.860282301902771, -0.06466065347194672, 1.0298908948898315, -0.8287069201469421, -1.2012022733688354, -0.8251984119415283, 0.6421324014663696, -0.8975695371627808, -1.4665374755859375, -0.9630705714225769, -0.16550210118293762, -0.1753896027803421, -1.5138047933578491, -0.10021933168172836, -2.20536470413208, -0.22264206409454346, 0.35267600417137146, -0.024025732651352882, 0.843290388584137, -0.69954514503479, 1.7357680797576904, -0.8966386914253235, 0.2019469290971756, 0.3513181209564209, -0.6637144088745117, -0.08332207798957825, -0.19797617197036743, 0.34636324644088745, 0.1357910931110382, 0.36793842911720276, 0.8208466172218323, 1.1033815145492554, 0.7473123669624329, -0.1858319193124771, -1.1733680963516235, -1.1670271158218384, 0.1956048160791397, 0.007006975822150707, -0.9468747973442078, 1.0882632732391357, -0.43344852328300476, -0.5407114624977112, 0.30490824580192566, -0.09745869785547256, -1.6001665592193604, 0.5994468927383423, -1.4822086095809937, -0.6218116283416748, 1.7682169675827026, -0.34109336137771606, 3.2443246841430664, 0.7482909560203552, -0.33786410093307495, -0.7066771984100342, -0.3228287994861603, 0.20650805532932281, 1.3326036930084229, 0.05252659320831299, -0.792974591255188, -0.6966437697410583, -0.28683051466941833, 1.065229892730713, 2.115283489227295, 0.670982301235199, 0.5830889344215393, -0.17732597887516022, 0.12398295849561691, -0.6458179950714111, -1.1318457126617432, 1.5070499181747437, -0.28381991386413574, 1.5412414073944092, 0.20712488889694214, -0.20395399630069733, -0.2747437655925751, 0.46800723671913147, -0.012659585103392601, -1.619630217552185, -0.23977310955524445, -1.2273731231689453, -0.5386210680007935, 0.7728943228721619, -2.028031349182129, -0.20837299525737762, 1.1141481399536133, 0.14353691041469574, -0.45784467458724976, -0.568291425704956, -1.4104843139648438, 0.018506888300180435, 0.43431076407432556, 0.3025672435760498, -0.8461731672286987, 0.16519102454185486, 1.7576335668563843, -0.4926013648509979, -0.8445136547088623, 1.2447803020477295, -0.25041571259498596, 0.41467535495758057, 0.1757722944021225, -0.21113425493240356, 1.5670806169509888, 0.6043298244476318, 0.32775747776031494, 1.0694695711135864, -0.5092695951461792, -0.9134579300880432, 0.4794694483280182, 0.12403115630149841, -0.15515175461769104, 2.24300217628479, -0.10624503344297409, 0.6425249576568604, -0.21549424529075623, -0.09927031397819519, 0.016332609578967094, -1.0061686038970947, 1.3279964923858643, -1.4577040672302246, 1.2366483211517334, 0.20023275911808014, 2.3804471492767334, 0.6581039428710938, -1.43660569190979, -0.787421703338623, -0.8609642386436462, 0.6384259462356567, 1.6948273181915283, 0.5183308124542236, 0.4713823199272156, 2.12039852142334, 1.5103048086166382, 0.5185402631759644, -0.5154152512550354, 0.6625757813453674, -0.8548564910888672, -1.1624900102615356, -0.3840419352054596, 0.29792800545692444, -0.5500771403312683, -0.05479821562767029, -0.350186288356781, 1.5082157850265503, -0.8707432746887207, 0.1886005848646164, 0.9619497656822205, 0.5703936219215393, -0.5537837147712708, 0.5232865214347839, 0.2661459743976593, 0.5123017430305481, 1.0485628843307495, -0.5087549090385437, -1.0109670162200928, 0.5544935464859009, -1.351848840713501, 2.239086151123047, 0.49798882007598877, 0.3809018135070801, -0.38222602009773254, 0.015828467905521393, 0.4380210340023041, 0.3409724235534668, 0.642255961894989, -0.18447871506214142, 0.6641280055046082, 0.6944830417633057, 0.5464816093444824, -0.4507675766944885, 0.6277378797531128, -0.0932156965136528, 0.58096843957901, 0.04610847309231758, 1.5739339590072632, 1.6154279708862305, 0.3921106457710266, -0.40105998516082764, -0.9026854634284973, 0.7263439893722534, -0.13144443929195404, 0.141793355345726, 0.4642784297466278, -0.12093411386013031, 0.9549046754837036, -0.2269631028175354, -0.5697043538093567, -0.3507803976535797, -0.24558289349079132, -0.40791237354278564, 0.21151652932167053, -0.011145046912133694, 0.41288086771965027, -2.1142935752868652, -1.6974810361862183, -0.14802664518356323, -0.46805018186569214, -0.7405100464820862, 1.085310459136963, -1.2754954099655151, -1.1959694623947144, -1.1924952268600464, 0.3144100308418274, 3.3174209594726562, -0.5675510764122009, -1.261917233467102, -0.13002918660640717, -0.3808104991912842, 0.36680248379707336, 1.0386844873428345, -1.6293169260025024, 0.803604006767273, -0.6535976529121399, 0.9613336324691772, -0.5799741148948669, -1.7108620405197144, 0.5707365870475769, 1.0266128778457642, 0.5829176306724548, -0.27684253454208374, 0.5342540740966797, 0.794061541557312, 2.138213634490967, 0.8493933081626892, -0.6579874753952026, 0.5908447504043579, -0.26032155752182007, 0.38944926857948303, 0.9284722208976746, 0.49287447333335876, 0.7809767127037048, -0.21247224509716034, 1.025258183479309, 0.8325747847557068, 2.270322799682617, -0.04210112616419792, 1.075891137123108, -1.2089521884918213, 0.2813519835472107, -0.8132548928260803, 0.7583550214767456, 0.22027529776096344, 1.2284128665924072, 0.9031422138214111, 0.7155226469039917, 0.44511300325393677, 2.0917935371398926, -0.03608812764286995, -1.853076457977295, 0.28915709257125854, 1.0526092052459717, -0.8119103312492371, 0.23911449313163757, -1.0519936084747314, -0.11386969685554504, 0.033462006598711014, -0.5205379724502563, -1.5631359815597534, -1.2233829498291016, -2.082406520843506, 0.3333154022693634, -0.6736322641372681, 0.2958953082561493, 0.4547232687473297, -0.5354223847389221, -0.685333788394928, -1.3619616031646729, -0.15422013401985168, -0.40113458037376404, 0.11849572509527206, -1.865410327911377, 0.572233259677887, -0.22264404594898224, 1.4538521766662598, 1.426990270614624, 1.353499174118042, -1.6835449934005737, 0.19651956856250763, -1.2754377126693726, -0.7581843137741089, 2.351362466812134, -2.2572011947631836, -1.4363120794296265, 0.5940626263618469, -0.343795508146286, 0.3406343162059784, -1.8106855154037476, 0.07548346370458603, -1.5127711296081543, -0.5779961347579956, 1.6128093004226685, -1.9693135023117065, 0.00549668911844492, -0.3344527781009674, -0.4716572165489197, 0.9236788153648376, -0.8823608756065369, 0.2902188301086426, -0.9013826847076416, -1.113008975982666, 0.10314369946718216, -1.7151243686676025, -0.7543724775314331, 0.3287901282310486, 0.024946631863713264, 0.9069757461547852, 3.1350975036621094, -0.45028024911880493, -0.7539933323860168, 0.6015579104423523, -1.5045326948165894, -0.9787930250167847, -1.7025665044784546, 0.6031579375267029, 0.5240939855575562, -0.38000044226646423, -0.15374886989593506, 1.1010228395462036, 0.33922573924064636, 0.7746585607528687, -0.11109314858913422, 1.9007960557937622, -0.6810481548309326, 0.7408551573753357, -0.16061224043369293, 0.6360713839530945, 0.29395726323127747, 2.267766237258911, 0.6938164830207825, -0.8988139033317566, 0.7703797221183777, 1.6062736511230469, -0.3401046395301819, 0.2523704171180725, 1.305857539176941, -0.7130410075187683, 0.10184590518474579, -1.3942756652832031, 0.26677629351615906, 0.8403580188751221, 0.08672061562538147, -0.923635721206665, -0.25280189514160156, 0.22869077324867249, 0.21043214201927185, 1.0981100797653198, -0.6365179419517517, 1.691489577293396, 0.3585455119609833, 0.3523457944393158, 1.8784843683242798, 0.07768166810274124, 0.5052802562713623, 0.05661022290587425, -2.3158202171325684, 0.8949018120765686, -0.3033192753791809, 0.6256242990493774, -1.0196114778518677, -0.28793877363204956, 0.9923312067985535, 0.4485224485397339, -0.7621268630027771, 1.942948341369629, -0.722093939781189, -0.18766829371452332, -0.7424384355545044, -1.5024348497390747, 0.04208122938871384, 1.1104199886322021, 1.5990630388259888, -1.8726557493209839, -1.5569748878479004, -0.012755122035741806, 1.1454715728759766, -0.6786212921142578, 0.692712664604187, 0.5610275864601135, -0.7888673543930054, -0.9660569429397583, 1.0914846658706665, -0.7257391810417175, 1.535984992980957, 0.5697326064109802, 0.2369159311056137, -0.8421707153320312, -0.5994310975074768, 0.16558535397052765, -0.9664177298545837, 1.1691856384277344, 0.16611793637275696, -1.2202657461166382, 0.41170409321784973, 0.27012211084365845, 0.3594886064529419, 1.0716469287872314, 0.204315647482872, -1.3642637729644775, -0.5590993762016296, -0.6197495460510254, 0.9907612204551697, 0.5171366930007935, 0.6355230212211609, 0.3538423180580139, -1.373716115951538, 0.25947290658950806, 0.3136194050312042, -0.2671213746070862, 1.0951052904129028, 1.6638535261154175, 0.08265047520399094, -0.27726882696151733, 0.31417012214660645, 0.30206459760665894, -0.48425403237342834, -0.4699758291244507, -0.2890060245990753, 0.949076771736145, 0.05443896725773811, 0.6080448031425476, 1.3431609869003296, -0.08461851626634598, 0.02850194275379181, -0.6243621110916138, -0.32693979144096375, 0.9380388259887695, -1.1294970512390137, 1.2171509265899658, 0.06910095363855362, 0.009542897343635559, -0.21846313774585724, 0.9761329293251038, 0.6781848669052124, -0.3421289026737213, 1.7827463150024414, -1.6926705837249756, -0.37458986043930054, 0.40845298767089844, -0.008944268338382244, -0.14547240734100342, -0.47125324606895447, 0.3090454638004303, 0.22675475478172302, 1.14226496219635, -0.213169664144516, 1.678366780281067, -1.0518929958343506, 0.3306606411933899, -0.08219670504331589, 0.20947016775608063, -0.36645153164863586, 1.0589817762374878, 2.228659152984619, 1.714028239250183, -0.2419629544019699, 0.8431117534637451, -1.0881037712097168, 2.0537455081939697, 0.19471804797649384, 1.4537160396575928, 1.2424290180206299, 1.1863877773284912, 0.12309898436069489, 0.7108763456344604, 0.9058574438095093, 0.9449343085289001, 0.7396056056022644, -0.49712073802948, -0.047294728457927704, -1.0537762641906738, -1.1119869947433472, 0.8199049234390259, 0.8297655582427979, 0.5684046745300293, -0.885243833065033, -0.29543259739875793, 1.0292918682098389, 1.7419425249099731, -0.31878209114074707, -0.06568458676338196, 1.0555602312088013, -0.6659910678863525, 1.021897792816162, -0.37350720167160034, -0.463398277759552, 0.12764473259449005, 2.0640058517456055, 0.4705505073070526, -3.3687732219696045, -1.4804526567459106, 0.2510721981525421, 0.44876059889793396, 1.1083711385726929, 0.19794535636901855, 0.5659818053245544, 0.410468727350235, 0.6289258599281311, -2.053244113922119, -1.2780267000198364, -0.485991895198822, 0.05483973026275635, -1.0677835941314697, -0.6980866193771362, 0.7705676555633545, 1.4417661428451538, 1.8807785511016846, -1.0731611251831055, -0.9570903778076172, 0.6850236058235168, -1.4941071271896362, -1.8257882595062256, 0.526699960231781, -0.18239474296569824, -0.30908501148223877, 0.5043390989303589, 1.1480505466461182, -0.16289976239204407, -0.3455444872379303, 0.31232061982154846, 0.7190624475479126, -0.7545985579490662, 1.4062426090240479, -1.0833224058151245, 0.1935100555419922, 1.5957353115081787, 2.376105308532715, -1.4177789688110352, 0.11412274092435837, 1.2797783613204956, -1.6037837266921997, -1.126481056213379, 0.038326770067214966, -1.3706353902816772, 1.696457862854004, -0.2114868462085724, -1.9690908193588257, 0.2823455035686493, 1.4236352443695068, -0.002984002698212862, 0.8492030501365662, 0.23224575817584991, -1.4129059314727783, -0.05157477781176567, 0.3355907201766968, -0.20195557177066803, -1.0565420389175415, 1.488431453704834, -0.49176904559135437, -0.040634412318468094, 1.3829433917999268, 0.06968890875577927, 0.5351531505584717, 0.46268779039382935, -0.024172404780983925, -1.2003735303878784, -0.2927311658859253, 0.03262550011277199, -0.33952975273132324, -1.6058391332626343, -0.5242413282394409, 0.3044274151325226, -0.3261030912399292, 0.18289931118488312, 1.6414597034454346, 0.02452073246240616, -0.7483734488487244, 0.16709068417549133, -0.5977481603622437, -0.8863844871520996, -0.18753491342067719, -0.29286012053489685, -1.344247817993164, -1.17526113986969, -1.8885263204574585, -1.0230779647827148, -0.22319744527339935, 2.234330177307129, -1.9573992490768433, -0.3388184607028961, -0.5465924739837646, 0.12533681094646454, -0.08087661862373352, 0.29621464014053345, 0.46372801065444946, 0.9387627243995667, 2.566401958465576, -0.810073733329773, -0.2699367105960846, 0.09339893609285355, -0.5123891830444336, 0.0868692696094513, -0.33412572741508484, -0.6248006820678711, 0.5723333358764648, -2.066089153289795, 1.0297939777374268, -0.81231689453125, -1.2646446228027344, -0.494099885225296, -0.7492943406105042, 0.8013148903846741, 0.22690965235233307, 0.671212375164032, 0.11864178627729416, 0.4994371831417084, -1.3146189451217651, 0.3211621642112732, -0.11370949447154999, 0.9576959609985352, 1.0503973960876465, -0.07015790790319443, -0.7662971019744873, -0.026941606774926186, -0.3021339476108551, -0.48583683371543884, -0.49748340249061584, 0.138192281126976, 1.005936622619629, -0.3566659092903137, 0.29715293645858765, 0.6251669526100159, -0.38971173763275146, 0.21278396248817444, -0.015556544065475464, -0.36536815762519836, 0.10799963027238846, -0.914991021156311, 2.5156612396240234, -0.7131187319755554, -0.7284543514251709, -0.12158055603504181, 0.8450366258621216, -0.4169224202632904, 0.29200929403305054, -0.7963096499443054, -0.18097926676273346, -0.8970048427581787, 0.2951361835002899, 1.6303802728652954, -0.27120065689086914, -0.8941612839698792, -0.25460824370384216, -0.696262001991272, 0.47607576847076416, 0.07754719257354736, -0.2046239972114563, -0.10725199431180954, -0.2555035352706909, -0.5483363270759583, -0.6490525007247925, 1.5102688074111938, -0.3160363733768463, -0.8925005793571472, -0.45899829268455505, 0.36550578474998474, 0.9055608510971069, -0.7737184762954712, -0.28407740592956543, -0.04280080646276474, 0.9403707385063171, 0.9848974943161011, -0.32267627120018005, -0.28232917189598083, 0.4302677512168884, 0.8673936724662781, 0.4041205644607544, 1.1935813426971436, 0.6340449452400208, -0.22119034826755524, -0.7661535739898682, -0.02489236555993557, -0.9620976448059082, -0.3201606869697571, 0.20956534147262573, 1.8064147233963013, 1.7697534561157227, -1.4909054040908813, -0.3581869304180145, 1.192294716835022, 1.7161054611206055, 0.6018662452697754, -0.12357491254806519, 0.1979091763496399, -0.250234454870224, -0.004763136152178049, -1.1872785091400146, 1.151361346244812, 1.2783006429672241, -0.2862913906574249, 1.1901333332061768, 0.5826478600502014, -0.253412127494812, 0.8652742505073547, 0.14668706059455872, 0.8179214596748352, -1.056806206703186, 0.5720392465591431, 1.5245624780654907, 0.37017542123794556, 0.9118697643280029, -0.6842182874679565, 0.842543363571167, 0.015375560149550438, 0.03645455092191696, -0.274468332529068, 1.5028185844421387, -0.6969342231750488, 0.018996847793459892, -1.1244521141052246, 1.6743736267089844, -0.4163497984409332, -0.3544439971446991, 2.085844039916992, -1.4080348014831543, 0.3259747326374054, 0.29478248953819275, -0.28475117683410645, 0.9474424123764038, 0.22517524659633636, 0.22462603449821472, -0.19318901002407074, 0.15607675909996033, -1.1512655019760132, -0.46990326046943665, -0.2808530330657959, -0.3272469639778137, 0.2416350245475769, 0.8760968446731567, 0.3788556158542633, -0.8992425203323364, -2.152846574783325, -1.0199809074401855, 2.5377824306488037, -0.3342599868774414, 1.5969158411026, 0.46745505928993225, -1.123885154724121, -0.16003388166427612, 0.2970474660396576, 0.14192719757556915, 1.5807337760925293, -0.620685338973999, -0.14592671394348145, 1.248405933380127, -0.5352194309234619, 0.9602144956588745, 0.6619852185249329, -0.6925860047340393, -1.9360761642456055, 0.8198613524436951, 0.5116304755210876, 1.0248759984970093, 0.15565839409828186, 0.7208930850028992, -0.503261923789978, 0.4217625856399536, 0.30508652329444885, -2.024301528930664, 0.06651235371828079, 1.302889347076416, -0.13274948298931122, 0.3736785352230072, 2.0450375080108643, -0.7729390859603882, -1.020268440246582, 0.5815706849098206, -2.159904956817627, 0.26932185888290405, 0.6289130449295044, 0.39253300428390503, 0.61539626121521, 0.13667722046375275, -1.185510277748108, 0.12510043382644653, -0.3299914002418518, 0.8644023537635803, 0.3500817120075226, -0.3011247217655182, -0.9410655498504639, -1.027309775352478, 0.4113122522830963, 1.0755536556243896, -0.13055261969566345, 0.3489361107349396, 0.783811628818512, -0.3627529740333557, -2.063629388809204, -0.43741074204444885, -0.9470558762550354, 0.15029025077819824, -0.03279772028326988, 1.7506942749023438, -1.9171133041381836, 1.4194343090057373, 0.09303741902112961, -1.8852263689041138, -0.7815566062927246, 1.101515531539917, 0.10382785648107529, 0.09129899740219116, -0.30939981341362, -1.8627227544784546, 0.16177617013454437, -0.3874306082725525, 0.5423218011856079, -0.002621479332447052, -1.010071873664856, -0.8669437766075134, 0.21502788364887238, -1.2064379453659058, -1.1423393487930298, -2.4373300075531006, 2.3158106803894043, 2.3063464164733887, -0.7766209840774536, -0.9127992987632751, -0.7639304399490356, -1.3317008018493652, -0.6312562823295593, 0.44885188341140747, -0.7148079872131348, 0.6554832458496094, 0.0002346668770769611, -0.9199396371841431, -0.5548267960548401, -1.1653488874435425, 1.5826698541641235, 0.3372431695461273, -0.8776929378509521, 0.703204870223999, -0.62518310546875, -0.9621248841285706, 0.5917192101478577, 0.48837363719940186, -0.5237393379211426, 0.5607276558876038, 0.9423331022262573, 0.17776641249656677, -0.5947225093841553, 0.07593043893575668, 1.3033543825149536, -0.3956410884857178, 0.10307905077934265, 1.1146947145462036, 2.2154459953308105, -0.5602558851242065, -1.0414432287216187, 0.6727285385131836, 0.4623841941356659, 1.0467896461486816, 1.050593614578247, -0.31608304381370544, -0.35777750611305237, 0.25436025857925415, 1.485433578491211, -0.759050190448761, -0.4473327100276947, -0.10620807856321335, 0.6222769021987915, 0.7280636429786682, -0.9064322710037231, -0.20878100395202637, -1.2451099157333374, -0.16975316405296326, -0.03236795961856842, 0.3148319125175476, -0.6580069065093994, -0.11035670340061188, 1.0842984914779663, 1.1075423955917358, 0.5184646248817444, -0.03245692327618599, -0.8477397561073303, 0.08897965401411057, 1.0019800662994385, 0.706292450428009, -0.6666775941848755, -0.5163620710372925, 0.1191028356552124, -0.7959827184677124, -0.5062541961669922, 1.2502299547195435, -0.04239868372678757, 1.4662139415740967, -2.4917380809783936, -0.6140751838684082, 1.5636345148086548, -1.0388606786727905, 0.41208967566490173, 0.8529685735702515, -0.6880725622177124, 1.0075113773345947, 1.5165432691574097, 0.6602714657783508, -0.2620692551136017, -1.336600422859192, -0.5179176330566406, -0.569471001625061, 0.7527673840522766, 0.6751678586006165, 1.6080985069274902, -1.1654223203659058, 1.0629762411117554, 0.844030499458313, 0.7529757618904114, -0.08685632050037384, 0.4441790282726288, 0.3828277289867401, 0.19447503983974457, -1.4535157680511475, -0.49495556950569153, 0.34336036443710327, 1.070267915725708, 1.5054211616516113, 0.8468118906021118, -0.0199293103069067, -0.7859100103378296, -0.7396853566169739, 1.058427333831787, 1.520974040031433, 2.0865983963012695, 0.5647114515304565, 0.24153678119182587, -0.928618311882019, 0.15810005366802216, -0.5898698568344116, -1.3371931314468384, 0.5288389325141907, -0.21868270635604858, 0.43897223472595215, 0.0852406844496727, -0.06072153523564339, 2.010073184967041, 0.6246607899665833, 0.1877540796995163, -0.5821658968925476, -1.068433165550232, -0.5914124846458435, 1.2156641483306885, -0.5216158032417297, 0.29416853189468384, -1.9071158170700073, -0.9421977400779724, 1.1097439527511597, 1.8122879266738892, -0.8929347395896912, 1.5762009620666504, -0.32512006163597107, -0.49325883388519287, -0.22308768332004547, -0.8402317762374878, -1.3750511407852173, 1.6822372674942017, -0.6136884093284607, 0.23037771880626678, 1.011853814125061, 0.05692493915557861, 0.17535650730133057, 0.8319661617279053, -1.350562572479248, -0.444413423538208, -0.9675615429878235, -0.1990726739168167, 0.5809844732284546, 0.13009671866893768, 0.1918030083179474, -0.7577194571495056, -1.566467046737671, 0.3147229552268982, -1.1134859323501587, -0.6186257600784302, -1.8292213678359985, -1.682167410850525, 2.213657855987549, 0.45873042941093445, -1.3509083986282349, 0.5409082770347595, -1.2424854040145874, 0.5379530787467957, 1.210491418838501, -1.9414105415344238, -2.0938901901245117, -1.050467610359192, 1.4763156175613403, -0.7029274702072144, -0.5846558809280396, 0.7142070531845093], [1.777753472328186, 0.7169538140296936, -0.765764594078064, -1.138655662536621, 0.5416698455810547, -1.949110507965088, 0.9066952466964722, 0.3726935386657715, -0.1763000339269638, -1.0626031160354614, 0.4298221170902252, -0.24247294664382935, 0.5574617385864258, -0.3056916892528534, 0.5088423490524292, 0.049898598343133926, 0.3377154767513275, -0.5904126167297363, -0.7189484238624573, -0.6109663248062134, 0.657365620136261, 0.4381576180458069, 0.659329891204834, 1.0648245811462402, -0.23975329101085663, -0.1943008154630661, 1.120962142944336, 0.0025968034751713276, -0.6477307081222534, 1.8891150951385498, -0.13419827818870544, 0.08353649824857712, 1.2210363149642944, 0.905768871307373, 1.039366602897644, -1.9482349157333374, 0.47687187790870667, -1.403752088546753, 0.7337109446525574, -0.9155330657958984, 0.6542014479637146, 1.0293689966201782, 0.8468919992446899, -0.7718509435653687, 0.7353583574295044, -1.271838665008545, -0.2375766485929489, -0.2313164323568344, 1.2930744886398315, 0.3694508969783783, -0.5580401420593262, -2.7609920501708984, 0.7448246479034424, -0.6140441298484802, -0.2756027579307556, -1.8825132846832275, 1.075455904006958, -0.2703110873699188, -0.012597348541021347, -0.8720030188560486, -1.466142177581787, 0.3559562861919403, -0.5718105435371399, 1.7414385080337524, 0.27855217456817627, -2.279193162918091, 0.014853380620479584, 1.211564302444458, -1.4122660160064697, 0.534152626991272, 0.5545279383659363, 0.48243919014930725, -1.8457493782043457, 0.8243417739868164, -2.1470577716827393, -2.1405043601989746, -3.4381730556488037, -0.27323195338249207, 0.6302427053451538, -1.7921830415725708, -0.6550989151000977, -1.2564789056777954, 0.9475668668746948, -1.480361819267273, 0.042231641709804535, 0.9973875880241394, 0.22876447439193726, 0.15866681933403015, 0.6348012089729309, 0.609219491481781, 0.7526499032974243, 0.3419928550720215, -0.04136684909462929, -2.1570143699645996, -0.48713481426239014, -0.5926404595375061, 0.3613915741443634, -0.5545545220375061, -0.7581482529640198, -2.392611026763916, 0.9115912318229675, -1.4972344636917114, 0.7035436034202576, -1.0640842914581299, -0.46405231952667236, -0.4743120074272156, 0.5638114213943481, 0.3359929919242859, 1.010182499885559, 0.08936357498168945, -1.372113823890686, -1.071395754814148, 3.242516279220581, -1.9379162788391113, 0.761526882648468, -0.3702670931816101, -1.467405080795288, -1.2008130550384521, 1.3660869598388672, -1.323235273361206, -0.47687989473342896, -0.986028254032135, 0.035510268062353134, 1.645448088645935, -0.5575295090675354, -0.0731881707906723, 0.8112112879753113, -0.4997807443141937, 0.03281141072511673, 0.674839198589325, 1.2670984268188477, -0.1640886515378952, -0.8331832885742188, -0.9857369065284729, -0.7690405249595642, -0.7747007608413696, 0.7538044452667236, 1.723140835762024, -1.3427258729934692, 0.6912459135055542, -0.9809815883636475, -0.9951208829879761, -1.579194188117981, -1.7856075763702393, 0.9447208642959595, 1.6873301267623901, 0.1460796594619751, 1.0348631143569946, 0.2484782189130783, 3.1575846672058105, 1.2566540241241455, 1.087849736213684, -0.4611615240573883, -1.4035950899124146, -0.14413800835609436, 0.7700896859169006, -0.47753724455833435, 2.1272668838500977, -0.3519260883331299, -1.3030078411102295, 0.6732467412948608, 0.22029872238636017, 1.1740033626556396, 0.3271835744380951, -0.8345977067947388, -0.9209312200546265, 0.264423668384552, 0.7653037905693054, -0.1261976659297943, 0.9918471574783325, -0.7797437310218811, -0.7890819311141968, -2.362419605255127, 0.21431902050971985, -0.2057110220193863, -2.5334486961364746, -0.11786668747663498, -0.8193399310112, -0.86913001537323, 1.7323554754257202, -0.916439414024353, -0.13344259560108185, -1.0000983476638794, -0.9428436756134033, 1.6537655591964722, -1.3844143152236938, 0.9551220536231995, 0.8597583174705505, -0.7395870685577393, 0.00537754176184535, -2.2370104789733887, 0.27810969948768616, 0.06046738103032112, 0.380360871553421, -1.1055845022201538, -0.5823954939842224, 0.07985437661409378, -1.5196099281311035, 1.1558032035827637, 0.16331173479557037, -0.9181455373764038, 1.622218370437622, -0.15916432440280914, 0.9916709661483765, -0.6726009845733643, -0.28087350726127625, 0.4227069616317749, 0.5095514059066772, 1.3430651426315308, 0.07106149196624756, 0.21837356686592102, 1.5335863828659058, 0.44673991203308105, 0.5863847136497498, 0.9270018339157104, 0.21893800795078278, 0.6364309787750244, -0.6270344257354736, -1.2611536979675293, -0.6324962973594666, 0.4254344403743744, -0.3725365102291107, 0.2568683922290802, 0.6672841310501099, 2.186553955078125, 0.9806919097900391, -0.18912915885448456, 1.0571492910385132, 1.2932876348495483, 0.5249003767967224, -0.7589748501777649, 0.43041303753852844, -0.29805389046669006, -0.38929665088653564, 1.69561767578125, 0.6520818471908569, 0.05944081395864487, -0.5788069367408752, 0.4595033526420593, 1.3701740503311157, 0.4988146424293518, -0.4570636749267578, 0.04912170022726059, -0.2704591155052185, 1.1195846796035767, 0.5290520191192627, 1.0984524488449097, 1.5020747184753418, 1.5696465969085693, -0.593252420425415, 0.11195668578147888, -0.21783289313316345, 0.15635724365711212, 0.11818048357963562, 0.5944728851318359, 0.5428547859191895, -0.9531026482582092, -1.6363790035247803, -0.5965753793716431, 0.37879595160484314, -1.3178892135620117, -0.5984945297241211, 0.8476487398147583, -1.110803484916687, 0.415494441986084, -0.4054713547229767, -0.09393750876188278, 0.2002631276845932, -1.061584711074829, -0.6933689117431641, -0.3349184989929199, 0.4427763521671295, 1.092335820198059, 0.7587818503379822, 0.7550430297851562, 0.8936084508895874, -1.7296967506408691, -0.6837730407714844, 0.7193447351455688, 1.783568024635315, -1.1722263097763062, 0.9462331533432007, 0.6470485329627991, -0.09449581801891327, 1.7285411357879639, 0.4220512807369232, -0.01066398061811924, -1.7513097524642944, 0.5760445594787598, -0.7573159337043762, -0.8014594316482544, -0.7683593034744263, -0.09355947375297546, 0.9692865014076233, -1.6417115926742554, -0.13032226264476776, 2.4869329929351807, -1.4867509603500366, -0.462357759475708, -1.707054615020752, 0.8948360085487366, 1.919238805770874, 0.11747971177101135, -0.23921075463294983, 1.1363047361373901, 0.16078564524650574, -0.21513357758522034, 0.07475893944501877, 0.3758543133735657, -0.7860647439956665, -0.6889224648475647, 0.5817831158638, -1.4738600254058838, -1.4038118124008179, 0.0433320552110672, -1.3548617362976074, -2.0067331790924072, 0.16013000905513763, -0.8214622139930725, -0.04682363569736481, -0.7296960949897766, 1.1456577777862549, 0.547311544418335, -1.0743855237960815, -0.06579142808914185, -0.9473893642425537, 0.9086454510688782, -0.26266855001449585, -1.240774154663086, -0.19151681661605835, 0.20725387334823608, 1.7644623517990112, -2.036857843399048, 0.020775267854332924, -1.3471380472183228, -0.9019688963890076, -0.4336560368537903, 0.5967621207237244, -1.0747610330581665, 0.013129966333508492, -0.9125221967697144, 0.10918085277080536, 0.18510471284389496, 1.4347096681594849, -0.34180426597595215, 0.7051438689231873, 0.5917482972145081, -0.3942320644855499, -1.2141823768615723, -0.16635778546333313, -0.4496510624885559, 0.6126853823661804, 0.2888830006122589, -0.4456360340118408, 0.5607503652572632, -0.10208902508020401, 2.1743197441101074, 0.16566957533359528, 2.0035560131073, 0.3741387128829956, 1.1318217515945435, 0.8544307351112366, 0.47410324215888977, -1.0030148029327393, -0.9916542172431946, 0.06934349983930588, -0.3640393614768982, -1.493398904800415, -0.10355328023433685, 0.13819009065628052, 0.4822864532470703, 0.09454188495874405, -0.15401488542556763, -0.5850920677185059, -1.2603119611740112, -0.3921491205692291, -0.25317999720573425, -0.06017618998885155, -0.71209317445755, 1.523347020149231, 0.44464609026908875, 0.4000149071216583, -0.4926997721195221, -0.10262436419725418, 0.4310348927974701, -2.3898239135742188, -1.1572500467300415, 0.873826265335083, 0.11659986525774002, 2.69279146194458, 0.07921347767114639, 2.0779874324798584, 0.13622136414051056, -0.778003990650177, -1.1485151052474976, 0.6623601317405701, 0.342876672744751, -1.2928613424301147, -0.7151467800140381, 0.7018517851829529, 1.5938875675201416, -0.7989203333854675, -2.161439895629883, -1.0399768352508545, -1.26749587059021, -0.6132558584213257, 0.23345112800598145, 0.12596294283866882, 1.2930978536605835, 0.41420528292655945, -0.00240722275339067, 0.7083854079246521, -0.9879196286201477, 0.49555501341819763, -0.23002305626869202, 1.5210374593734741, 0.46808522939682007, -0.02207653783261776, 1.4171310663223267, -1.1120344400405884, -0.8072017431259155, -0.557196319103241, -1.4731239080429077, -1.2126171588897705, 0.7512086629867554, 0.06776706874370575, 1.1818948984146118, 2.106112241744995, -1.4886560440063477, -2.0118634700775146, 1.7199617624282837, 1.6861552000045776, -0.14038300514221191, 1.1719077825546265, -0.6375890374183655, -0.7614316344261169, -0.002091060858219862, -0.5934773087501526, -1.7873634099960327, 0.8475194573402405, -0.7913968563079834, -0.856880784034729, -1.5761975049972534, 0.15744414925575256, -0.1294882893562317, 0.8892737627029419, -0.1878664642572403, 1.4136862754821777, 0.8335916996002197, 2.263859748840332, -0.49937108159065247, -0.2188626229763031, -0.4304128587245941, 1.3747395277023315, 0.7716633081436157, 1.4215022325515747, 0.6769629120826721, 1.2151497602462769, 0.4908345937728882, 0.8965563178062439, 1.203307867050171, -3.488572835922241, 0.5716382265090942, 1.6384737491607666, -1.3026374578475952, -0.18255163729190826, 0.09131055325269699, -0.6694765686988831, 0.6417370438575745, -0.08130082488059998, -1.0562994480133057, -1.5947766304016113, 0.7956242561340332, -1.1056959629058838, 1.6741933822631836, -0.302166610956192, 0.24578078091144562, -1.370908498764038, -1.5037909746170044, 0.515561044216156, 0.46337229013442993, 1.5872745513916016, -1.2220439910888672, -0.5943314433097839, -1.2313880920410156, -2.289886236190796, -0.5983309745788574, -1.6106959581375122, 0.8008583784103394, 0.8644685745239258, 1.0978957414627075, -0.5324901342391968, 1.4661173820495605, 0.029342681169509888, -0.5555101037025452, 0.25228235125541687, 0.2682228088378906, -0.6718709468841553, 0.22324128448963165, -0.85821932554245, -0.08693903684616089, 1.8363763093948364, -1.3693737983703613, 0.04070606827735901, -0.6696842908859253, 0.1508995145559311, 0.12627170979976654, 1.9485905170440674, -0.5519431233406067, 1.673561692237854, 0.01490681804716587, 0.8517340421676636, 0.04695405811071396, 1.4817522764205933, 0.18661244213581085, 0.013557231985032558, -1.930171251296997, -0.3454004228115082, 1.0450870990753174, 0.03191255405545235, 1.167253017425537, -0.32024678587913513, -0.26175224781036377, -0.6698535084724426, -2.466129779815674, 1.22916579246521, 1.07624089717865, 0.8633462190628052, 0.36365607380867004, -0.22859378159046173, -0.9090104699134827, 0.5670839548110962, 0.7585832476615906, 1.6242663860321045, 0.11791092157363892, -0.38469722867012024, 0.17622774839401245, 2.0918149948120117, 0.448131263256073, 0.7824641466140747, -0.8341068625450134, 0.6276748180389404, 0.49371999502182007, 0.0034049709793180227, 0.7883515357971191, 0.7672486901283264, -1.035388708114624, -0.14127381145954132, -0.6627094149589539, 1.7990139722824097, 1.7497738599777222, 0.6404951810836792, -0.034371353685855865, 0.5194096565246582, -0.29216864705085754, -0.5530285835266113, 0.030090322718024254, 2.06672739982605, -0.15584467351436615, 0.9426576495170593, -0.07113577425479889, -0.28975996375083923, -0.31871190667152405, 1.1806597709655762, -0.6122123003005981, -1.0532082319259644, -1.1518115997314453, -1.2470935583114624, -0.017931580543518066, -0.3508433997631073, -0.44197407364845276, -0.8679291605949402, 1.0581557750701904, 0.5272908210754395, 0.5500290393829346, -0.4609781801700592, 0.26223474740982056, 0.6689837574958801, 0.5535824298858643, 1.2210335731506348, -0.3242611885070801, -1.1266565322875977, 1.2814592123031616, 1.5603752136230469, 0.6428607106208801, 0.4060231149196625, 1.095177173614502, -0.06433133780956268, -0.011179773136973381, 0.42045214772224426, 1.0986206531524658, 0.5718016028404236, 1.1117874383926392, -0.958853542804718, -0.7112963795661926, -1.157733678817749, 0.6731306314468384, -1.0908986330032349, -0.12274745851755142, -0.8853853940963745, 1.1276189088821411, -1.0455524921417236, -0.26481541991233826, 0.8643391728401184, 0.2292189747095108, 0.06292745471000671, -0.17060601711273193, -1.7162816524505615, -0.29896703362464905, -0.03504382446408272, -0.6008399128913879, -2.1162126064300537, -0.9996299743652344, -0.10451030731201172, 1.143216609954834, 0.7694761753082275, -1.104625940322876, 0.6687546372413635, 2.1070339679718018, -1.060610294342041, 0.8704380393028259, 0.5253177285194397, -1.4347732067108154, -0.7276525497436523, -0.49629655480384827, -0.22903525829315186, 0.11761952191591263, 0.4820154905319214, 0.9073640704154968, -1.897300362586975, 0.18512704968452454, -0.3229416608810425, -1.6047885417938232, -1.2372444868087769, 0.5105867981910706, -0.8710333704948425, 1.3918715715408325, -1.5267949104309082, 2.023501396179199, 0.8367711305618286, -0.9159632921218872, 0.30088672041893005, 0.24137678742408752, -0.10268387943506241, -0.5017070174217224, 0.3541519343852997, -0.07248122990131378, 0.3311406672000885, 0.7572005391120911, 0.538936972618103, 1.1000686883926392, -0.1939464658498764, 0.9356678128242493, 0.8247339129447937, -0.2714844048023224, -0.6688582301139832, 1.4761399030685425, 2.4659647941589355, 0.9007308483123779, -0.41924646496772766, 1.3769655227661133, 0.8510749340057373, -0.1545598953962326, -2.0481224060058594, 0.08169221878051758, -0.9903736710548401, 1.1051955223083496, -1.031309962272644, 1.1213291883468628, -0.451680988073349, 0.14714109897613525, -1.3912450075149536, 0.49810317158699036, -0.20244622230529785, -0.8862677216529846, -0.9681848287582397, 0.6761230826377869, -1.5664477348327637, 0.0990314856171608, -0.04832641780376434, -0.8663438558578491, 0.7534584403038025, -1.0669282674789429, -1.00702702999115, -1.3838847875595093, 1.7000197172164917, -0.7445565462112427, 0.7977850437164307, 0.8873322010040283, 1.1909751892089844, 0.8253888487815857, -0.4820026755332947, 1.7280879020690918, 0.2159876525402069, -0.7558373808860779, 0.22172078490257263, -0.22047147154808044, 0.6889118552207947, -0.22398093342781067, -1.2045831680297852, -1.6728572845458984, -0.6218672394752502, 1.2976001501083374, 1.4721291065216064, 0.3475324511528015, 0.7895738482475281, -0.6830469965934753, 0.15013709664344788, -1.5814288854599, 0.43790289759635925, 0.578011691570282, 1.9433542490005493, 0.02544328197836876, -3.045433282852173, -0.13455837965011597, -2.326709032058716, 1.586368203163147, 0.97118079662323, 0.8446683883666992, -0.2671639025211334, -1.628235101699829, -0.3073345422744751, 0.41648417711257935, -1.5067262649536133, -0.0269493255764246, 1.3661576509475708, -0.5682792067527771, -1.2011524438858032, -0.4987357556819916, 0.14634542167186737, 0.6172055006027222, -1.4460150003433228, 1.3950541019439697, 0.1095452606678009, 0.2969980835914612, -1.7031546831130981, 1.9698898792266846, -1.0687850713729858, -1.247674822807312, 0.008561818860471249, 0.384625107049942, -0.9519541263580322, 0.6085441708564758, 0.7259835004806519, -0.13795216381549835, -0.7368770837783813, -1.9214481115341187, 1.7957690954208374, 2.243792772293091, 0.32030636072158813, -0.2432526797056198, -1.2890071868896484, 0.30854588747024536, 1.438486933708191, 0.2773612439632416, -0.749458372592926, -0.6387744545936584, -1.2895853519439697, -0.849410891532898, -0.6215693950653076, -0.6864898800849915, -1.205024242401123, 0.9688114523887634, 0.7134192585945129, 1.5677070617675781, 0.7655429840087891, -0.43436747789382935, -0.9341832995414734, -0.22695934772491455, -0.0981970801949501, -0.47514116764068604, -0.2897135615348816, 1.213561773300171, 0.45252320170402527, 1.2708685398101807, 2.0191218852996826, 0.20950748026371002, 0.6983155608177185, -1.0312374830245972, 0.294172078371048, 0.28960981965065, 1.0612215995788574, -0.9695137143135071, -0.24295492470264435, -1.2893413305282593, 1.1054608821868896, -2.2477171421051025, -1.634076714515686, 0.040639478713274, -1.0751603841781616, -0.40472230315208435, -2.6899771690368652, -2.0406739711761475, -1.160456657409668, 0.5000860095024109, -0.47531792521476746, 0.3744406998157501, -0.8879379630088806, -0.444009006023407, -0.7489132881164551, 0.34203943610191345, 1.9503865242004395, 0.8649861812591553, 0.3672337532043457, -1.157081127166748, -0.27755066752433777, -0.34207087755203247, -0.07869818061590195, 0.5051969289779663, -0.44600847363471985, -1.2838809490203857, 1.8302968740463257, -0.5417518019676208, 0.36567580699920654, -1.2002290487289429, -1.4477534294128418, -0.6088868975639343, -0.28299131989479065, 0.264322429895401, -0.17706343531608582, 1.1839619874954224, 1.0999046564102173, 0.25606247782707214, 1.4415351152420044, -0.01606334187090397, 0.9088780879974365, 0.254944771528244, -1.5156642198562622, -0.447278767824173, 0.9677718281745911, -0.5469444394111633, -0.5441969037055969, 0.06689484417438507, 0.6610620617866516, -0.7985566854476929, -0.4229707717895508, -0.5196216106414795, 0.383501797914505, 0.3320942521095276, 0.18366895616054535, -1.5500584840774536, 0.06033018231391907, -0.4163389205932617, 0.4140253961086273, 0.02279900200664997, 0.27226772904396057, 0.14264138042926788, -1.0683605670928955, -0.12296780198812485, 0.1739136129617691, -0.03983932361006737, 0.5501375198364258, -0.6172676682472229, 0.1834431290626526, -0.7295701503753662, 0.21390818059444427, 2.354203701019287, -0.829578697681427, 0.7136921882629395, -0.22707465291023254, 0.0027958382852375507, -1.2352761030197144, 0.5577294230461121, -0.4997888207435608, 0.7623668909072876, 0.19594061374664307, -1.5986037254333496, 0.3462376892566681, 0.44323989748954773, 0.3388282358646393, 0.5386233925819397, -1.1064980030059814, -1.8006558418273926, -0.1823963075876236, 0.6946420669555664, 1.600081443786621, -0.1448618620634079, 0.8605550527572632, -0.36644741892814636, 0.6862747073173523, -2.5974574089050293, -1.135984182357788, 0.5926522016525269, -0.21712344884872437, 0.7004621028900146, 1.0606871843338013, 0.5430557131767273, 0.9387167692184448, -0.4487009644508362, -1.987122893333435, 0.16333703696727753, 0.23297914862632751, -0.4904404878616333, -1.2772746086120605, -0.37100401520729065, -1.1866267919540405, 0.8066422343254089, -0.6094657182693481, -1.2318190336227417, -0.5395002365112305, -1.4114857912063599, 1.0610711574554443, -1.31972336769104, 0.6104320287704468, 0.12439446896314621, -1.7159591913223267, -0.7494520545005798, -0.02304902859032154, 1.596616506576538, 0.7941336631774902, 0.8122079968452454, 0.27235686779022217, 1.1354597806930542, 0.8857810497283936, -0.18885095417499542, -0.5362986326217651, 0.7021699547767639, 0.24085664749145508, -0.42636731266975403, 0.8339843153953552, 0.8154146075248718, 0.6865102648735046, -0.12862394750118256, 1.0602869987487793, -0.05846196413040161, -0.4640265703201294, 1.3662793636322021, 0.9221138954162598, 0.4023585915565491, -0.5342940092086792, -2.8099658489227295, 1.1477354764938354, -0.9059920907020569, 1.654258370399475, 0.45828840136528015, 0.5338693857192993, -0.8612157702445984, 0.6989836096763611, -0.9813399314880371, 0.29357051849365234, 1.0727964639663696, -0.2339736968278885, -0.18811428546905518, -1.5346723794937134, 0.5797366499900818, 1.7212382555007935, -0.0801071748137474, -0.6036410927772522, 0.45889753103256226, -1.7808057069778442, 1.7098294496536255, -0.35124462842941284, 0.5747352838516235, 2.493103504180908, 0.936055600643158, -1.9221731424331665, -0.09861020743846893, -1.2185808420181274, 0.5332479476928711, 1.2391533851623535, 1.7798149585723877, -0.1113680824637413, 0.2950587570667267, -1.3140003681182861, -0.20600855350494385, 0.046377092599868774, -0.48988643288612366, 0.9976359605789185, -0.611678421497345, -2.7823545932769775, -0.33659127354621887, -0.09968244284391403, 0.10708460956811905, 0.1051836684346199, -0.9354674816131592, -0.07291887700557709, 0.19631721079349518, -0.2697868347167969, 1.2724339962005615, 0.233968585729599, 2.2842371463775635, 0.8753388524055481, -0.5197159647941589, -1.4581018686294556, 1.6659597158432007, -0.36199289560317993, -0.949926495552063, -1.8924375772476196, -0.8241569399833679, -0.7662971615791321, -0.5191885232925415, -1.9308239221572876, 0.19360238313674927, -2.2378673553466797, 4.6130512600939255e-06, 0.20582611858844757, -0.24310597777366638, -2.039053201675415, -0.2055448740720749, 2.8510305881500244, 1.635602593421936, -0.22626709938049316, -1.5456656217575073, -0.12495700269937515, 1.3160011768341064, -0.6150728464126587, 0.8225823640823364, -0.04405234754085541, -1.6040441989898682, 0.49266284704208374, 0.38489511609077454, 0.757678747177124, 0.8321301937103271, 0.42179539799690247, 0.7101057767868042, -1.0814175605773926, 1.3788429498672485, -1.2685043811798096, 0.7617504596710205, 0.2859451472759247, -0.25911322236061096, -0.39238160848617554, -0.4272891581058502, 0.02728533372282982], [-0.5632123351097107, -0.6245155334472656, 0.7404285669326782, -0.4130227267742157, 0.2636699974536896, 1.0775315761566162, 0.09764710068702698, -1.1052143573760986, 0.5540657043457031, 1.5293502807617188, -0.33338260650634766, -0.605233371257782, -0.2662970721721649, 1.2149815559387207, 0.2490144670009613, 1.3962148427963257, 0.24200856685638428, 0.40527644753456116, -0.07986552268266678, -0.6492872834205627, 0.41110026836395264, 0.11044196039438248, 0.057757340371608734, -0.31197589635849, 0.13651514053344727, 1.4808143377304077, 1.0206650495529175, 0.5040238499641418, -1.4011460542678833, -0.9247713088989258, 0.76759934425354, -1.2168811559677124, -2.0926003456115723, -2.1039297580718994, 0.2018849104642868, -0.3577529191970825, 0.6828386783599854, -0.30870291590690613, 0.062134403735399246, -1.2997411489486694, -0.33018699288368225, -0.19477716088294983, 1.495197057723999, 1.6924495697021484, 0.33690163493156433, 0.16286392509937286, 0.27442818880081177, -0.7717633843421936, 0.7258654236793518, -0.29597026109695435, -1.0498336553573608, 1.912921667098999, -0.7955713868141174, 0.3633453845977783, -0.07365497201681137, -0.8627526164054871, -0.08154385536909103, -1.0671117305755615, -0.16295582056045532, 0.04932147264480591, -0.2609562873840332, -0.42985379695892334, 1.9835376739501953, 1.4709583520889282, -0.02766481600701809, 0.3244711458683014, -2.339493989944458, -1.3240230083465576, 0.5978766083717346, 0.8933985829353333, 0.5653015375137329, 0.8912194967269897, -1.2941452264785767, 0.3611151874065399, 0.46699807047843933, -0.27557840943336487, 0.9448787569999695, 1.1995214223861694, -0.6023358106613159, -0.8201682567596436, -1.0226542949676514, 0.8142591118812561, 0.25857794284820557, 0.5329738259315491, -0.23379071056842804, 0.3444151282310486, -0.20445740222930908, -0.27515074610710144, -1.13718843460083, -0.029112352058291435, -0.7055652141571045, -0.5859916806221008, 0.4204190671443939, 0.4989088773727417, -0.05454271659255028, -0.44143909215927124, -1.161460518836975, 0.9267263412475586, 1.635908842086792, 1.2807166576385498, 0.018716657534241676, 0.5526078343391418, -0.6083942651748657, 0.7536462545394897, -1.221052646636963, -1.868008017539978, 0.35996347665786743, 0.3579615652561188, 0.4756479859352112, 0.9294635057449341, -1.9390356540679932, -1.408694863319397, 1.4840725660324097, 0.08027207851409912, -0.7824782729148865, -0.039856597781181335, -1.390953779220581, 1.4448961019515991, -0.6937226057052612, -0.961224377155304, -1.2627488374710083, -1.6472216844558716, -0.5756646394729614, 0.7763077020645142, 0.3806455433368683, -0.8217695355415344, 0.4460124969482422, 0.3431834280490875, 0.08097927272319794, -0.34989312291145325, 0.18177060782909393, 0.3097646236419678, 0.2092771679162979, -1.7134549617767334, 0.7817758917808533, 1.0745140314102173, 1.8027570247650146, 1.0301532745361328, 1.3897371292114258, -0.3547424376010895, -0.14615613222122192, 0.5855692028999329, 0.9761615991592407, 0.2928258776664734, -1.0456238985061646, -0.616051197052002, -0.09619388729333878, -2.4134581089019775, -0.27206286787986755, 0.3873771131038666, 0.22276389598846436, 0.511780321598053, -0.8192587494850159, -1.4827450513839722, 0.38719090819358826, -1.159216284751892, 2.905855894088745, -0.04377409815788269, -0.3304259777069092, 0.9363495111465454, 1.6752221584320068, 1.886088252067566, 1.727954387664795, -1.4628043174743652, -0.2630377411842346, 0.13616906106472015, -0.006720250006765127, -1.430108904838562, 1.4888765811920166, 0.28441181778907776, 0.10702519118785858, -0.8739374279975891, -1.1997450590133667, -1.5807359218597412, 0.8300161361694336, 0.6528521180152893, 0.7830697894096375, -1.8453090190887451, -0.06765047460794449, 0.49703094363212585, -0.629791259765625, -0.17339515686035156, 0.8602244853973389, 1.0763425827026367, 1.0128893852233887, -1.553954839706421, 0.653518795967102, 0.7319092750549316, -2.1248464584350586, -0.15837933123111725, -0.6477460265159607, -1.0562357902526855, 0.4467276334762573, -0.08304969221353531, 0.3592158257961273, 0.6975350975990295, 0.8054177165031433, -0.013689501211047173, 0.3373931050300598, -0.8489293456077576, -0.2908364236354828, -0.8870550394058228, 0.34811916947364807, -0.717123806476593, 0.5332115888595581, -2.1802585124969482, 0.5774267315864563, -0.5232905745506287, -0.19300143420696259, 0.4342171549797058, 0.14223499596118927, -1.7558332681655884, -1.1661139726638794, 1.3332948684692383, 0.8003605008125305, -1.0270856618881226, 1.9968302249908447, -0.8981061577796936, -1.0102850198745728, -0.3835262656211853, 0.2981197237968445, 1.3886029720306396, 0.43200117349624634, -2.7173664569854736, -0.30857521295547485, -0.23031051456928253, 0.24763646721839905, 0.8378247618675232, 0.5183120369911194, -0.2266128659248352, 1.4879792928695679, -0.7838181257247925, 0.6431554555892944, -0.051033783704042435, 0.4615955948829651, -0.6227362751960754, 0.7963072657585144, -0.6265474557876587, -0.6834325194358826, 0.6111486554145813, 0.9693400859832764, 1.0531984567642212, 0.35769084095954895, 0.41968685388565063, -1.1338984966278076, -1.7187608480453491, -1.468617558479309, 0.33609431982040405, 0.4912067651748657, -0.6438930630683899, 0.6177331209182739, -0.5460571050643921, 0.7261919975280762, 1.558035135269165, 1.2679535150527954, -1.0481361150741577, -0.31304359436035156, -0.08781340718269348, 1.0843356847763062, -0.6230204105377197, 0.8474871516227722, 0.6868430972099304, -0.18162229657173157, 1.2171834707260132, 0.5568690896034241, 2.8313167095184326, -1.1074460744857788, 0.7509547472000122, 0.008947549387812614, -1.5821822881698608, -0.9099805951118469, -0.27705511450767517, -1.5025160312652588, 2.35737943649292, -0.769568920135498, -1.1740041971206665, 0.609286904335022, 0.725711464881897, -1.2191697359085083, 1.9456982612609863, 1.1076953411102295, -0.7070163488388062, 1.898464322090149, 0.5862147808074951, -0.6226300001144409, -0.2645165026187897, -0.6671671271324158, -0.37728753685951233, 1.055021047592163, 0.3472804129123688, 1.4853469133377075, 0.4364945590496063, -1.7204482555389404, -0.21653299033641815, 0.013067762367427349, -1.2861500978469849, -0.15315350890159607, 0.836581289768219, 0.2623427212238312, -3.6284279823303223, 0.6389548182487488, 0.5130236148834229, 0.4392511248588562, 0.39094385504722595, -1.435455560684204, -0.2627337872982025, 1.222877025604248, -1.4518262147903442, -0.7782614231109619, 0.6442228555679321, -0.6882354617118835, 0.8471422791481018, 0.8163278698921204, -0.7590224146842957, -1.3084003925323486, -0.1832750290632248, -0.466081827878952, -1.5125691890716553, -0.461012065410614, 0.5783207416534424, 0.004250699188560247, -1.3193577527999878, -0.7427271008491516, 0.3099146783351898, -0.4503883123397827, -1.2742422819137573, -0.03568292036652565, -0.2200329750776291, 0.29808753728866577, -1.8527164459228516, -0.06596887111663818, 0.34323734045028687, 0.9663700461387634, -0.7792037725448608, 0.15335892140865326, -0.25157585740089417, -7.631757034687325e-05, -0.16069385409355164, 1.053274154663086, -0.09849198907613754, -0.2673547863960266, -0.7876497507095337, 0.28474369645118713, -1.7127145528793335, 0.35335278511047363, 1.1209299564361572, 1.0926672220230103, 0.700702428817749, 1.1139616966247559, -0.9725810885429382, 0.3408357799053192, 0.25141745805740356, 0.1661655753850937, 0.6605964303016663, 0.6657931208610535, -1.3514820337295532, 0.32459983229637146, 0.4282461106777191, 1.2154115438461304, -1.6714022159576416, -0.45430949330329895, 0.19840605556964874, 0.6804724931716919, 1.1529581546783447, -0.031142577528953552, -0.10189467668533325, -0.7471661567687988, 0.10985104739665985, -2.017939329147339, -0.7963610291481018, 0.6420608162879944, 1.3391882181167603, -0.1480443775653839, -0.2028476744890213, 0.7528676986694336, -0.18620721995830536, -0.7883834838867188, -0.4054821729660034, -0.2817719280719757, 0.9007118344306946, 0.5936791300773621, 1.0831589698791504, -0.8953304290771484, 0.6097304224967957, 0.49048885703086853, -0.7872418761253357, -0.42001909017562866, 0.311023086309433, 0.4009537100791931, -0.4723334014415741, 0.367549866437912, -0.07296966016292572, -0.8689864873886108, -1.038365125656128, 0.9355729222297668, -0.8301180005073547, -1.6123933792114258, -1.6226861476898193, 0.5852481722831726, 0.0797114223241806, 1.4420958757400513, 0.27115216851234436, 1.4635792970657349, 0.48721733689308167, 1.6291568279266357, 0.7127715945243835, 0.6165516376495361, 1.5340605974197388, 0.5497193336486816, 2.290196418762207, 0.5830538868904114, -0.9675214886665344, 0.4420977234840393, 0.06772256642580032, -1.3728485107421875, 0.9886219501495361, 2.349916696548462, 0.9511682987213135, -0.6833818554878235, 0.6216515302658081, -2.0585312843322754, -2.381974458694458, -0.8048635125160217, 0.7660008668899536, -0.023454826325178146, 0.537868082523346, -0.613702654838562, -0.8896379470825195, 0.682572066783905, 0.0897665023803711, -0.9490180611610413, -0.6911207437515259, -0.9759508371353149, -0.3279235363006592, 0.7455064654350281, -0.7413002252578735, -2.06825852394104, -2.009782552719116, 0.6370564699172974, -1.4645845890045166, -0.5853397250175476, 0.24058832228183746, -0.12116186320781708, 0.646294355392456, -0.47548410296440125, 0.44897156953811646, -0.38781657814979553, -0.14609484374523163, 0.42459043860435486, 0.8230746388435364, -1.1145670413970947, 0.09316881000995636, 0.0778970718383789, 0.43512052297592163, 0.14962445199489594, 2.078754186630249, -0.4607345461845398, 0.09329139441251755, -0.9111855626106262, 0.07934407144784927, 1.5060807466506958, -0.2783359885215759, -0.17685675621032715, 0.8047884702682495, -0.9323320388793945, 0.5948600769042969, 1.1743115186691284, 1.0475953817367554, 1.581373691558838, -0.0384659469127655, -0.16832156479358673, -0.7610195875167847, 0.0021258036140352488, 0.6185336709022522, 0.6073024868965149, -0.2331685572862625, -0.15183955430984497, 0.35141950845718384, -1.1695911884307861, -2.1103615760803223, -0.2822209596633911, -1.1468229293823242, 0.15008093416690826, -0.9298904538154602, -0.4004592001438141, 0.7559863924980164, -0.6381607055664062, -0.33804774284362793, -0.22181794047355652, 1.3423902988433838, -0.3709697425365448, 0.7439119219779968, -0.3913239538669586, 0.6170482635498047, 0.5997260808944702, -1.3025263547897339, 0.9083810448646545, 0.9002665877342224, 0.12185675650835037, -0.49241092801094055, -0.8597273230552673, 1.203833818435669, 0.11278218030929565, -1.2817250490188599, -0.4483773410320282, -1.407912015914917, 0.8368610739707947, 1.4313013553619385, -0.7183537483215332, -0.08085399121046066, 0.6041265726089478, -1.678297996520996, -0.4746510684490204, -0.48606419563293457, 0.10083980858325958, 0.2180953174829483, -0.7942284345626831, -0.8105669617652893, -0.8670791983604431, -0.843483567237854, -0.5367100238800049, 1.1303399801254272, -0.9380711913108826, -1.9882813692092896, 0.42477190494537354, 1.1768141984939575, -0.35993650555610657, 0.5015585422515869, 0.2609367072582245, 0.4729987382888794, 1.3698254823684692, 0.7007812857627869, 0.6117978096008301, -2.5407497882843018, -0.7961151599884033, 0.5817959904670715, -0.7129544019699097, 0.0745915099978447, 0.6711675524711609, -0.24117900431156158, 0.008817954920232296, -0.19149287045001984, 1.4126744270324707, 0.6851826906204224, -0.7149287462234497, -0.765918493270874, -0.7588009834289551, -0.8845095038414001, -0.4266749322414398, -0.7145306468009949, -0.4152345657348633, 0.0704331174492836, -0.4816093444824219, -0.40712979435920715, -0.5480524897575378, 0.9972848892211914, -0.4812624156475067, 0.42136090993881226, 0.9173623919487, 0.42737630009651184, 0.4766172766685486, -1.6952241659164429, 1.1632453203201294, 1.833526849746704, 0.1422766149044037, 0.5054652094841003, -2.0525732040405273, 1.1627987623214722, -0.126693457365036, 1.8654676675796509, -0.49626925587654114, 1.924752950668335, 0.7217915058135986, -0.07203321903944016, 1.052876591682434, -1.5033297538757324, -0.5993830561637878, 0.26388019323349, -0.3313407003879547, 0.45750734210014343, -0.20182311534881592, 1.9497674703598022, 1.0488128662109375, -0.6319903135299683, 0.47908928990364075, -0.365957111120224, -0.12438856810331345, 0.35416024923324585, 0.6241896748542786, -0.4139612317085266, -0.4681940972805023, 0.7867389917373657, -0.19631525874137878, -0.452605664730072, 0.6980294585227966, 0.7218387722969055, -0.8819398880004883, 1.6051318645477295, 1.6540902853012085, 2.6418066024780273, -0.4266672432422638, 1.8437734842300415, -0.06108225882053375, 0.6069251894950867, -1.277425765991211, -0.994822084903717, 0.7456974387168884, 1.4363152980804443, -0.7048293948173523, 0.5580214858055115, 0.7368496656417847, 1.262679934501648, -0.4484370946884155, -0.8968506455421448, 0.7666727304458618, -1.729487419128418, -0.1138521283864975, 1.9697173833847046, -1.012467384338379, -0.4693107604980469, -0.23144567012786865, 0.29861682653427124, 0.7911459803581238, 1.0236977338790894, -0.5572078227996826, 1.0316219329833984, -1.2876033782958984, 1.552628755569458, 0.15687371790409088, 0.6863114833831787, -0.40726691484451294, -0.31631988286972046, -1.0479522943496704, -0.9289528131484985, 0.011853625066578388, 0.16942793130874634, -0.12491835653781891, -0.2097337245941162, -0.5068235993385315, -0.7091944217681885, -0.18776832520961761, -0.4059985876083374, -2.049492835998535, -0.37975195050239563, -0.05307776853442192, -1.5127445459365845, -0.2986806333065033, 0.7930954694747925, -0.566156268119812, 1.4893168210983276, -1.5913450717926025, 0.6406962275505066, -0.4816426932811737, -1.725842833518982, -0.3823191821575165, 0.3695887625217438, -1.769415259361267, 1.2044353485107422, -2.3251264095306396, 0.5010934472084045, 0.9028594493865967, -1.3277281522750854, 0.8489609956741333, -0.9842498898506165, -1.3740729093551636, 0.5959263443946838, -0.5569648146629333, -1.1149206161499023, -0.6519957184791565, -0.8522499203681946, 1.5426191091537476, -0.5552681088447571, -0.6847352981567383, 0.9379348754882812, -0.6949249505996704, -1.7311220169067383, -0.6266646385192871, 0.08720628917217255, 0.7623566389083862, 0.1688765287399292, -0.5776944756507874, -0.1516573578119278, -0.8171405792236328, -0.15876485407352448, -0.9577693939208984, 0.8721447587013245, -1.8706014156341553, 1.2762054204940796, -0.06979186087846756, -0.39836424589157104, 0.5902854204177856, -0.7820951342582703, 0.4313996732234955, 0.8539028167724609, 0.531647801399231, -1.3640732765197754, 0.31012406945228577, 0.21667782962322235, -0.14961449801921844, 0.5226477384567261, 1.0746182203292847, 0.2948957085609436, 0.2848951518535614, 0.18817296624183655, 1.3935046195983887, -1.0129894018173218, -2.069978952407837, 1.2943012714385986, -0.29564058780670166, -3.0700337886810303, 0.9360383749008179, -1.4625176191329956, -0.8878611326217651, 0.8247765898704529, 1.1094757318496704, 0.6298390030860901, -0.9798164367675781, 2.832508087158203, 0.42503467202186584, 1.4481182098388672, 0.9586902260780334, 0.06240459904074669, -0.032287295907735825, 0.8527565598487854, 1.275599479675293, 0.6986597180366516, -1.1551977396011353, -1.1166465282440186, -0.5188725590705872, -1.6073625087738037, -1.3062268495559692, -0.6188657879829407, -0.923130452632904, 2.6071012020111084, -0.44289326667785645, 1.2759082317352295, 0.4341496229171753, -1.4600539207458496, -0.7501049041748047, -1.217855453491211, 1.509031891822815, 0.47917497158050537, 1.3241370916366577, 1.3023849725723267, -0.0442366898059845, 0.5928436517715454, 0.35281068086624146, 0.18141265213489532, -0.7965265512466431, 0.45674118399620056, 0.2635680139064789, -1.601306676864624, 0.6098843812942505, 0.1860284060239792, -0.7534657120704651, -1.433422327041626, -0.49964243173599243, -0.9089810252189636, 0.3884291648864746, -0.5381379127502441, 0.22681313753128052, 0.5906725525856018, 0.3004322648048401, -1.659058928489685, -0.23893658816814423, -0.8701448440551758, 0.6332691311836243, -2.3932578563690186, 0.43145298957824707, 1.4252954721450806, 0.006940597668290138, 1.2816839218139648, 1.6251955032348633, -0.7309690713882446, -1.7005506753921509, -1.848514437675476, -0.17866726219654083, -0.7981319427490234, 0.3298285901546478, 0.4549281597137451, -1.2540465593338013, 0.4591370224952698, -0.4589375853538513, -2.4430580139160156, -0.9350515604019165, -0.14717236161231995, -1.1308594942092896, 0.7146462202072144, 0.3500252962112427, -0.3600143492221832, 0.7022051215171814, 1.8969224691390991, 0.14419348537921906, -0.5540078282356262, 1.4744571447372437, -0.9598263502120972, 0.1479235142469406, 0.888403594493866, -0.8385626077651978, 0.8830185532569885, -0.41542571783065796, 0.058556269854307175, -0.11208619922399521, 0.1121789962053299, 0.5155525207519531, 0.5804344415664673, 0.7603450417518616, -2.611699104309082, -1.007685661315918, -1.7246249914169312, -0.36839377880096436, 0.21785782277584076, 1.31120765209198, 2.0212669372558594, 0.5370438098907471, -0.13035288453102112, -0.28231537342071533, 0.24142524600028992, 1.386449933052063, 0.9184060096740723, 0.731879711151123, 0.6469432711601257, -0.5372804999351501, -0.15593640506267548, 0.25522905588150024, -1.3031041622161865, -0.36573341488838196, 0.20221781730651855, 0.02894783578813076, -0.5189894437789917, 0.7961203455924988, 0.551900327205658, -0.9326273798942566, -1.5031315088272095, 0.14529405534267426, -0.3429759740829468, 0.1637217104434967, -0.4056507349014282, 0.14338988065719604, 0.012798618525266647, 0.2255142629146576, -1.5564724206924438, 0.5405100584030151, 0.5949212312698364, 0.8053215742111206, 0.18267859518527985, -0.7960870862007141, -1.0700081586837769, -0.1345907300710678, 0.6816202402114868, -0.18568404018878937, 0.8396571278572083, 0.20068833231925964, 0.8157362937927246, 1.3138760328292847, 0.14363788068294525, -0.5299047827720642, 0.36053645610809326, 0.7624190449714661, 0.08274673670530319, 0.7950265407562256, 2.570168972015381, -0.1558743566274643, 0.5209562182426453, -1.693673014640808, -0.7141930460929871, -0.2518005967140198, 0.7673864364624023, -1.1531410217285156, 0.6445121765136719, 0.47264575958251953, 1.3963156938552856, -1.6060881614685059, 1.3938250541687012, -0.001473437063395977, 2.152381420135498, 1.8956849575042725, 2.0454323291778564, 0.20770327746868134, 1.5979366302490234, 1.9257935285568237, 0.9849212169647217, 0.032522980123758316, -0.7479894757270813, 0.6150007843971252, -2.256765604019165, 0.05455293506383896, -0.5509587526321411, -0.22985778748989105, 0.5371792316436768, 0.13351108133792877, -0.6782164573669434, 0.32133492827415466, 0.05010579153895378, 0.5292915105819702, 0.7289326786994934, -0.703872799873352, 0.6194502711296082, -1.1658231019973755, 0.5283758640289307, -1.4762835502624512, 2.3080410957336426, -0.9964637756347656, -0.5630660057067871, 1.825590968132019, -1.2726820707321167, -1.7995370626449585, 0.9576242566108704, -0.29049423336982727, 1.0448286533355713, 0.24704551696777344, -1.6295591592788696, -0.9367029070854187, 1.4209562540054321, 1.7493300437927246, -2.416168451309204, 1.5462539196014404, 1.3835746049880981, -1.3322733640670776, 2.0991692543029785, 0.4205399751663208, -0.9661528468132019, -0.388060063123703, -0.3558521270751953, 0.37269681692123413, -0.4614419639110565, 0.36276838183403015, 0.3973276913166046, -0.7711138725280762, 1.7992578744888306, 0.6204755306243896, -1.8486230373382568, -0.558075487613678, -0.8445737957954407, -0.2370559573173523, -0.43056467175483704, 0.3203485608100891, -0.35802552103996277, -0.48208311200141907, 0.9604797959327698, -0.6499052047729492, -0.374027281999588, 0.3920140862464905, 0.5428543090820312, 0.46781492233276367, -0.1531481295824051, 0.11074259877204895, 2.5048317909240723, 0.8278030157089233, 1.5360549688339233, -0.3089272975921631, -0.8828572630882263, 2.2337496280670166, -0.051688987761735916, -0.43942007422447205, 0.34739813208580017, -1.4818035364151, -0.06495694816112518, -0.22556738555431366, 2.1081666946411133, -1.1266335248947144, 0.21029984951019287, 1.3364803791046143, 0.379239946603775, 1.7919421195983887, -1.0109435319900513, -2.0158350467681885, -0.9265106916427612, 0.7683512568473816, -1.0438774824142456, 0.15160587430000305, -0.6190379858016968, 1.1069679260253906, -1.7473366260528564, -0.913621187210083, -0.19967015087604523, -0.5664718747138977, 1.3944487571716309, 1.1018640995025635, -0.26857370138168335, 0.9159815907478333, -1.5662022829055786, -0.4672389328479767, 1.9159471988677979, -0.34090256690979004, 0.09532217681407928, 1.4301707744598389, 1.8943332433700562, -0.6287897229194641, -0.267624169588089, -0.5461523532867432, -0.9588094353675842, -1.8358842134475708, 1.1206958293914795, -1.9411060810089111, -0.29837390780448914, -0.06613707542419434, -0.910041868686676, 0.003169774543493986, 0.7653860449790955, -0.1933080554008484, -0.043351467698812485, -1.05579674243927, 1.517970085144043, 0.8947751522064209, -0.7979806065559387, -0.19079838693141937, 1.0973565578460693, 0.8847394585609436, -0.6992069482803345, -0.6059428453445435, -1.3028696775436401, 1.1342581510543823, -1.1342041492462158, -1.4339755773544312, 0.2531183063983917, 0.6151677370071411, 0.5815899968147278, -0.19326314330101013, -0.291801780462265, -1.1076761484146118, -0.49976950883865356, -0.15254193544387817, -1.2153531312942505], [1.5633472204208374, 0.7109103202819824, 3.365787982940674, 1.3005404472351074, -0.40854182839393616, -1.3352572917938232, 1.968741536140442, -0.5109992623329163, -0.8457016944885254, 2.0193657875061035, -2.3052425384521484, 1.3530856370925903, -3.1559555530548096, 0.8650428056716919, 0.21891596913337708, -0.035260893404483795, 1.2409147024154663, 0.4252331852912903, 0.03421498090028763, 1.5171757936477661, -0.414813369512558, -0.6915390491485596, -0.6427513360977173, -0.33215081691741943, -1.0260752439498901, -0.6095543503761292, -0.22381187975406647, 1.2467552423477173, 0.8839423060417175, 0.18680301308631897, 0.5432985424995422, -0.17122112214565277, 0.5981466174125671, -0.07372748106718063, 0.9319127798080444, -1.7307424545288086, -0.9575652480125427, 0.03164402022957802, 0.5637307167053223, 1.8102797269821167, -0.3046585023403168, 0.5156862139701843, -0.9444910883903503, 0.22135469317436218, -0.5158759355545044, -0.02859596349298954, 1.0675638914108276, 1.0659056901931763, -0.6632976531982422, -0.49633562564849854, -1.7312653064727783, -0.14993424713611603, -0.9905416369438171, 2.1331379413604736, 1.501802921295166, 0.472829669713974, 0.8402345180511475, -0.32216230034828186, -0.025786515325307846, 0.6944692730903625, -0.10622096806764603, 2.0643186569213867, -0.42188796401023865, 0.6627899408340454, 0.9261412620544434, -1.142311692237854, -0.4803873896598816, 0.5280557870864868, -0.8612743616104126, -0.971639096736908, 1.690477967262268, -0.6919649839401245, -1.7783887386322021, 1.4913493394851685, 0.3134327232837677, -0.44324222207069397, -0.018509797751903534, -1.2811787128448486, -0.7728781700134277, -0.18190929293632507, -0.0894865021109581, -0.16867737472057343, 0.8842851519584656, -0.3477121591567993, 0.013015084899961948, -1.1438688039779663, -1.1212066411972046, -0.10798192769289017, 2.2961769104003906, 0.44922295212745667, -0.5686917304992676, 0.3344559669494629, 1.1285103559494019, -0.5431607961654663, 0.005213230382651091, -1.7855088710784912, -1.488881230354309, -0.06379860639572144, -0.7866875529289246, 0.12696309387683868, -1.8076211214065552, 0.6740366816520691, 0.9859941005706787, 0.061803799122571945, -0.09552676975727081, -0.44436392188072205, -0.10747608542442322, -0.1357705295085907, -1.941678762435913, -0.48635637760162354, -0.6161990165710449, 0.36515864729881287, -0.36343875527381897, -1.338246464729309, -1.0006576776504517, 0.7815898060798645, -0.013506009243428707, -0.4949444830417633, 0.5023119449615479, 0.9986597895622253, 1.202623724937439, 0.3768408000469208, 0.5844171047210693, 1.4354132413864136, -0.5122926235198975, -0.876861572265625, -0.7961816191673279, 1.5406080484390259, -0.5618661046028137, 0.8328571915626526, -0.7461125254631042, -1.3058263063430786, 0.31416428089141846, 1.202056884765625, 0.4489474892616272, 0.11188651621341705, 0.62103670835495, -0.0247665848582983, 0.9010201692581177, -0.9439888000488281, -0.6680220365524292, 0.14171157777309418, -0.08317801356315613, -0.7446573972702026, 1.833024501800537, 1.6189403533935547, -1.3059184551239014, 0.5739536285400391, -0.5874893069267273, 1.0841014385223389, 1.191929578781128, -1.01193368434906, 0.12776780128479004, -0.7147080302238464, -1.3876127004623413, 1.2138097286224365, -0.13878701627254486, 0.5006359219551086, 1.7556966543197632, 0.5959110856056213, -0.2671114504337311, 0.7098438143730164, 0.7440415620803833, -0.7088063955307007, -0.21460793912410736, -0.9525834321975708, -0.9019960761070251, -0.7732947468757629, 0.9716780781745911, 2.400139331817627, 0.39278194308280945, -0.11278048157691956, 1.2152769565582275, -1.814948320388794, 0.4797549843788147, 0.2708094120025635, -0.4926205575466156, -0.2202436476945877, 0.12116345763206482, -1.6639024019241333, 1.4199334383010864, -0.5998531579971313, 1.7726088762283325, -1.7659516334533691, -0.5200290679931641, -1.165971040725708, 0.17075833678245544, -0.28785446286201477, 0.43223732709884644, -0.6687465310096741, 0.33282920718193054, -1.5255728960037231, -0.11648783832788467, -0.8946114778518677, -0.9285094738006592, 0.04209258407354355, -2.2262299060821533, 0.8769147992134094, 0.06765197962522507, -0.8796329498291016, 2.338754653930664, -0.6943917870521545, 0.4889894723892212, 0.26031309366226196, 0.38213029503822327, 0.06201254948973656, -0.31510254740715027, -0.2947998642921448, -0.8122437000274658, 1.2714813947677612, -1.11447012424469, -0.5811820030212402, 0.5949522256851196, -0.046877890825271606, 0.23457784950733185, 0.5743547081947327, 0.199744313955307, 0.41054096817970276, 0.6569979786872864, 0.6086075901985168, -0.9876354932785034, -0.41419529914855957, 0.06091875955462456, 0.7550684809684753, -0.3491761088371277, -0.6182560324668884, -0.890545129776001, 0.014930392615497112, -0.7901888489723206, 0.9877902269363403, 0.529060423374176, -0.39895492792129517, -0.49838560819625854, 0.28598669171333313, 0.6405729055404663, 0.29662641882896423, 0.8999612927436829, 0.7255062460899353, 0.7761532068252563, 0.00011572791845537722, 1.2782350778579712, -1.6276202201843262, 0.3167548179626465, 1.0158028602600098, 1.765234112739563, -0.0021019268315285444, 1.3160362243652344, -0.5760102868080139, -0.4473082721233368, 0.3883059024810791, 1.387605905532837, 0.857648491859436, -0.8440467715263367, 0.0476384237408638, -0.6143366098403931, -0.19374686479568481, -0.39803192019462585, 1.6987959146499634, -1.4394252300262451, 0.6692612767219543, -1.3543118238449097, 1.3331451416015625, -0.47828903794288635, -1.1753264665603638, 0.7542978525161743, 0.40312862396240234, 0.20070311427116394, -1.2072972059249878, 1.2906991243362427, -0.6799128651618958, 0.7642225623130798, 0.6973393559455872, -0.39543911814689636, -0.7678745985031128, -0.50296950340271, -0.539484977722168, 0.37499675154685974, 1.4581290483474731, 0.9728013873100281, 0.7463291883468628, 0.6197004318237305, -1.5570800304412842, -0.642107367515564, -0.2840298116207123, 0.5975900292396545, -0.4253827631473541, 0.6855357885360718, -0.47546151280403137, 0.09088962525129318, -0.7928423285484314, -0.020909903571009636, 0.15576449036598206, -0.5237273573875427, 0.13668663799762726, -1.3554257154464722, 3.200201988220215, -1.0789958238601685, -0.01474805735051632, -0.6118261814117432, -0.3455889821052551, 0.5756405591964722, -0.9972864389419556, 0.3847099840641022, 1.4300658702850342, 0.0592319630086422, 0.41903820633888245, 1.0542218685150146, 1.2287544012069702, 0.2523207366466522, -0.4125141501426697, -0.03300290182232857, -0.1698470115661621, 2.1752476692199707, -0.7719256281852722, -0.9354494214057922, 1.5147721767425537, 0.02842629700899124, 0.4687606394290924, -0.26563453674316406, 1.9071481227874756, 3.724989175796509, -1.5231037139892578, 0.7159481048583984, -0.20823927223682404, -0.1489827185869217, -0.33611994981765747, -0.3863593637943268, -0.3115984797477722, 0.8124088048934937, 0.7948020100593567, 0.504888653755188, -1.1755329370498657, 1.30647611618042, 2.2124087810516357, -0.28929468989372253, 0.8154584169387817, 0.8707789778709412, 0.2682786285877228, -2.0056073665618896, 0.9872258305549622, 1.267371654510498, 0.15684223175048828, -0.4942491352558136, -0.7296449542045593, -0.4593923091888428, 0.4024186134338379, -0.624599814414978, -0.5723731517791748, -2.1070234775543213, 0.9560345411300659, 0.7779260873794556, 1.4233258962631226, 0.6278283596038818, 1.278322458267212, -0.42955493927001953, 0.8792024254798889, 0.1602538526058197, -0.08360067754983902, -2.754300594329834, -2.0406060218811035, 0.06515052914619446, -1.5570555925369263, 1.1455652713775635, 0.30169153213500977, 0.35826703906059265, 0.021958963945508003, 0.5318338871002197, -0.40564367175102234, -0.4409067928791046, 0.9051608443260193, 0.09876551479101181, -1.7959274053573608, 0.5096789002418518, -1.4158518314361572, -1.3053687810897827, -0.4993724524974823, 0.622390866279602, 1.3390709161758423, -0.4449309706687927, -0.3167601227760315, -0.9096822142601013, -1.3338778018951416, -0.5437785983085632, 0.6789653897285461, -1.6215345859527588, -0.1972012221813202, 1.428823471069336, 0.3195728063583374, 2.262035608291626, 0.1876010000705719, -0.05617647245526314, 0.2875344753265381, 0.8150668740272522, 0.000548452022485435, -0.37496814131736755, 0.03290282562375069, 0.6396008729934692, 0.6287046670913696, 0.4235246479511261, 0.5101154446601868, 0.8033933043479919, 0.9898289442062378, 1.311111569404602, -2.224480152130127, -2.3080241680145264, 0.1168348640203476, -0.7779520153999329, -0.3479589521884918, 0.814030647277832, 0.2794604003429413, -1.9865796566009521, 1.661427617073059, -0.5357678532600403, -0.2480919510126114, 1.0705701112747192, 0.2762832045555115, -0.22509023547172546, -0.6838201880455017, 0.180770605802536, 1.8288589715957642, -1.4584712982177734, -0.1331188678741455, 0.013113442808389664, 0.5807868242263794, -1.0268515348434448, 1.1192512512207031, -0.26918837428092957, -1.2123593091964722, 2.082296371459961, -0.9685896635055542, -1.3420283794403076, -0.13994261622428894, 0.3560384511947632, 1.4741194248199463, 1.5466771125793457, -0.6849448680877686, 1.1935046911239624, -0.6718177199363708, -0.35134026408195496, -0.31950175762176514, 1.0828155279159546, -0.011022564955055714, 1.6113007068634033, -0.07835324853658676, 0.20587608218193054, 0.2250678688287735, -1.2882270812988281, -0.9402680397033691, 0.07292937487363815, -1.0987492799758911, 1.3778334856033325, 0.5888625979423523, -0.11220958828926086, -0.4754842519760132, 1.477475881576538, 0.642307460308075, 0.07526913285255432, -1.9073623418807983, -1.077569603919983, -0.7926071882247925, -1.1056115627288818, 0.23410429060459137, -1.4888843297958374, 0.010194780305027962, -1.0634684562683105, 1.4223030805587769, 1.0387632846832275, -0.38474300503730774, -1.1065213680267334, -1.003849744796753, -0.48502224683761597, -0.5392656922340393, 1.6378451585769653, 0.7895116806030273, 1.1492711305618286, -1.4456266164779663, -0.7322826385498047, 0.11891800910234451, -1.1508578062057495, 1.0066823959350586, -0.025866758078336716, 2.069624423980713, -0.00040409481152892113, -1.2012081146240234, 0.6971261501312256, -0.07863757759332657, 1.9556479454040527, 0.2373192459344864, -0.7567141652107239, -0.8486360311508179, -1.5292719602584839, 0.39724719524383545, -2.0758376121520996, 0.7452899813652039, -0.056636568158864975, 0.8006226420402527, -1.1035550832748413, 1.373302936553955, 0.43131083250045776, 0.5160382986068726, 0.5437308549880981, 0.08126134425401688, 1.492613434791565, 0.6712782979011536, 0.5643801689147949, 1.6862623691558838, 0.2694699764251709, 0.1744750440120697, 0.17613419890403748, -0.37797069549560547, 1.3614706993103027, -1.209546446800232, -1.2902570962905884, -2.1043505668640137, -0.7315783500671387, 0.09778470546007156, 1.2241188287734985, -0.08272621780633926, 1.1401805877685547, -0.498505175113678, 1.2964308261871338, 0.855376660823822, 0.011300794780254364, -0.011906650848686695, 1.667243480682373, -0.49061208963394165, 1.6354985237121582, -1.2820138931274414, 0.772723376750946, -0.0941314846277237, 2.2953765392303467, -0.7848549485206604, 0.3879256248474121, -0.32436901330947876, 0.27299195528030396, 1.0317803621292114, 0.5866482853889465, 1.3215367794036865, 0.25493139028549194, -0.7380964756011963, -0.016642041504383087, -1.1386057138442993, 0.480370432138443, -0.5454223155975342, 0.9046898484230042, 0.5254644155502319, -1.2381197214126587, 2.152170419692993, 0.8374024033546448, -0.1494603306055069, 1.938628077507019, 0.11617837101221085, 0.15038661658763885, -0.8896880745887756, -0.8999570608139038, -0.3597663640975952, 1.4166806936264038, 0.15620571374893188, -0.055913347750902176, -1.5083041191101074, -1.301155686378479, 0.07001105695962906, -1.6254196166992188, -1.794895052909851, 0.1690012365579605, 1.1790152788162231, -0.5490021109580994, 0.7161927223205566, -0.5382041335105896, 1.2997965812683105, 1.1141563653945923, -0.6880401372909546, 0.2999555766582489, -0.2927042841911316, -0.8533995747566223, -0.321935772895813, 2.458383083343506, -0.25378069281578064, -0.3720996379852295, -0.031252216547727585, -1.3563014268875122, -1.1491860151290894, -0.17764435708522797, -0.6961403489112854, 0.8316848278045654, -0.7779842019081116, 0.6909389495849609, 2.19600248336792, -0.09157313406467438, -1.5635943412780762, 0.02831966057419777, -0.3036002516746521, 0.30163654685020447, 0.5513825416564941, 0.20103275775909424, -0.9994964003562927, -0.5159342288970947, -1.153444528579712, 0.5155107378959656, -0.5083822011947632, 1.0124139785766602, 0.5365323424339294, 0.07474493235349655, -1.2808396816253662, -0.4544479250907898, -1.3731838464736938, 0.15227150917053223, -0.35092082619667053, 0.0759371817111969, 0.35371819138526917, 0.6177710294723511, 0.3654342293739319, -0.4139329493045807, -2.078399896621704, 0.038694828748703, 0.031097201630473137, -0.1750800907611847, 1.604217767715454, -0.39804789423942566, 0.9297580718994141, -0.06126480549573898, -0.5510123372077942, -0.11394254118204117, -1.2154370546340942, -1.5279725790023804, -0.9461943507194519, -0.8117877244949341, 0.2068338245153427, -1.4948729276657104, -0.6062836647033691, 0.03755189850926399, -0.6589797735214233, 0.6389788389205933, -0.17858333885669708, -0.7489759922027588, 0.24612432718276978, 0.8255716562271118, -0.44047975540161133, -0.760417103767395, -0.04590775445103645, -0.21161706745624542, -1.4232524633407593, 1.4493331909179688, 0.5738937854766846, 0.5557804107666016, 1.3446546792984009, 1.9970602989196777, -0.4993405342102051, -0.1839940994977951, 0.34083911776542664, -1.6628867387771606, 0.7732239961624146, -0.2368975430727005, -0.4476013779640198, 1.0728678703308105, -1.1379257440567017, 0.6550697088241577, 0.8828577399253845, -1.2434085607528687, 0.4460344910621643, -0.34458133578300476, 1.2180064916610718, 1.17686927318573, -0.9477023482322693, 0.5831969976425171, -0.19053632020950317, 0.10895632207393646, -1.285861611366272, -0.4850296080112457, -0.27081942558288574, 1.590024471282959, -1.7385863065719604, 0.04268454760313034, 0.36632040143013, -0.7161200046539307, -1.2953873872756958, 0.39786210656166077, -0.19676640629768372, 0.472658634185791, 1.508331298828125, -0.11507421731948853, 2.043121576309204, -0.613555371761322, 1.0580741167068481, 0.8908321857452393, -0.64711993932724, -0.9854103326797485, -1.355745792388916, 0.7255247235298157, -0.14687439799308777, -0.9707092046737671, -1.2347493171691895, 0.923144519329071, -0.04207070916891098, 1.1278985738754272, -1.1245081424713135, -0.7310089468955994, 0.7701660394668579, 0.08440493792295456, -1.086002230644226, 0.49052903056144714, -0.29116255044937134, -1.0898927450180054, -0.16102173924446106, -1.4148625135421753, -1.1653039455413818, 0.7959755659103394, 0.8691678047180176, -1.017738699913025, -1.3390668630599976, -0.8155794739723206, -0.43780088424682617, -1.4496042728424072, -0.17297112941741943, -1.0411179065704346, 0.9645426273345947, 0.33506786823272705, 0.14299294352531433, -0.5847140550613403, -0.27713900804519653, 0.8208094239234924, -0.41685789823532104, 0.4193483293056488, -0.1319231241941452, -0.1261960119009018, 1.7662091255187988, -0.6699349880218506, 1.0158336162567139, 0.5908191800117493, 0.4512443542480469, -0.20942473411560059, -1.2499936819076538, -0.4004576504230499, -1.2471650838851929, -1.0096654891967773, -1.2135260105133057, 0.004334417171776295, 0.3946056663990021, 0.2673851251602173, -1.4248895645141602, 2.0855507850646973, -0.5026984214782715, -0.420207679271698, -0.8582533001899719, -0.05288643762469292, -0.4821431338787079, 0.08943280577659607, 0.8757038712501526, 0.7564241290092468, -0.9392136931419373, 0.025562049821019173, -0.4482753574848175, 1.659449577331543, -1.8564790487289429, -0.4491029977798462, -0.005930802784860134, 0.3835828900337219, 0.9602595567703247, -0.6478201150894165, -2.015817403793335, -0.041575752198696136, -1.9765523672103882, 0.8168904185295105, 1.031424641609192, -1.0940450429916382, 2.0504391193389893, 0.6018727421760559, -0.6986019611358643, 0.6409944295883179, -0.6739969253540039, -1.5194429159164429, 0.44824540615081787, 1.3048975467681885, 0.6585071086883545, -0.7295776009559631, 1.320037603378296, 0.7123090028762817, -1.0335676670074463, 1.5006766319274902, 1.4209332466125488, -1.624639630317688, 0.8488199710845947, 0.5701154470443726, 1.031042456626892, -1.5478867292404175, 0.9665256142616272, -1.3231195211410522, -0.22431500256061554, -0.12954944372177124, -0.19284766912460327, 0.8146001696586609, -0.22257421910762787, -0.11358009278774261, -0.46983256936073303, 0.3522811532020569, 0.3595068156719208, 1.0113275051116943, -0.35179731249809265, 0.8745443820953369, -0.23951901495456696, -0.3594696521759033, -1.4344125986099243, -1.162558913230896, 0.6241487264633179, -1.1881155967712402, 0.22474727034568787, -0.4174421727657318, 1.919559121131897, -0.07172758132219315, -0.9407318234443665, 1.0550609827041626, -0.8003587126731873, 1.550244927406311, 0.6190974116325378, -1.0672863721847534, 0.3223898112773895, 0.5035268664360046, 0.673267662525177, -0.20280395448207855, 0.009464376606047153, 1.6748402118682861, 0.5287492871284485, 0.464385449886322, 0.7926323413848877, 0.9349720478057861, -2.230938673019409, -1.4385069608688354, -1.4329440593719482, 0.03725415840744972, -0.11104246973991394, 1.5875656604766846, 0.5480780601501465, 0.5381421446800232, -0.019401248544454575, -2.5965216159820557, -0.9096502065658569, 2.337775707244873, 0.0004666628083214164, 0.3976666331291199, 1.2887324094772339, 0.4761359393596649, -0.7112147808074951, -1.3730741739273071, -0.18241222202777863, 0.5659874677658081, -0.7541602849960327, -1.899574637413025, 0.9293398261070251, 1.6304253339767456, 1.4199917316436768, 1.9246500730514526, -0.32020124793052673, 0.6343929767608643, -1.3604828119277954, 3.2528281211853027, 2.26102876663208, 0.4013349711894989, 1.3212159872055054, 0.7436740398406982, 0.23051780462265015, 1.6713002920150757, -0.6194906830787659, 1.190514326095581, 0.7884164452552795, 1.357429027557373, -0.8851778507232666, 1.0798715353012085, -0.565078616142273, -0.34835150837898254, 0.7416784167289734, 1.9285670518875122, -0.37339869141578674, 0.20419789850711823, -0.43174394965171814, -0.011418696492910385, -1.4188079833984375, 0.8321544528007507, -0.9834623336791992, -0.8683608174324036, -0.21263791620731354, -0.06475988030433655, -0.40157774090766907, -1.076723337173462, 1.519918441772461, -0.4510064423084259, 0.6623327136039734, -0.1525818258523941, 1.5421808958053589, 0.6186997294425964, 0.09860275685787201, 1.4999594688415527, 0.8058607578277588, -0.029683653265237808, -0.18029671907424927, -1.0813621282577515, 0.38009119033813477, -1.2524951696395874, 0.1295822262763977, -0.8099370002746582, 0.4530055522918701, -0.5072540044784546, -0.3431992530822754, -1.9697926044464111, 1.1452090740203857, 0.01886897161602974, -0.3413549065589905, 0.1510159820318222, 1.5403330326080322, 0.19216763973236084, 1.7171223163604736, -0.15107494592666626, 0.5532719492912292, -1.2464998960494995, 0.6643243432044983, -0.7015804648399353, -1.0606744289398193, 1.1965910196304321, 0.5914089679718018, 1.2206178903579712, -2.010836362838745, 0.10861854255199432, -0.048138707876205444, 1.1110312938690186, -0.2243579924106598, -3.1135430335998535, -1.7868406772613525, 0.6189013719558716, 0.2981923520565033, -0.12518008053302765, -0.5816129446029663, -0.733357310295105, 1.7836871147155762, 1.4502975940704346, -0.05820534750819206, -0.39696526527404785, -0.5778511762619019, 0.7981292009353638, -0.3290715515613556, 0.429741770029068, 2.125467300415039, 0.7305858731269836, -0.3331797420978546, -0.3769776523113251, -0.7572883367538452, -0.2602405548095703, 1.074947476387024, 0.6611561179161072, 1.1397982835769653, 0.42426592111587524, 0.2241784632205963, 2.196408748626709, -0.5594673156738281, 0.04750007390975952, 0.5543893575668335, -0.09526918828487396, 0.9026166200637817, 0.6957445740699768, 0.5450119376182556, 1.0833510160446167, -1.449191689491272, 0.9093673229217529, -1.3998525142669678, 2.258134603500366, -0.627170205116272, -0.7414615154266357, -0.7208064198493958, 0.2224092036485672, 0.9922311902046204, 0.08031362295150757, 0.2280389815568924, 1.4288989305496216, -0.24155321717262268, -0.45974698662757874, 0.3450944721698761, -1.0084270238876343, 1.4620589017868042, 0.0332881398499012, -0.13940639793872833, -0.20016588270664215, -0.41359564661979675, 0.8489072918891907, 0.1612943857908249, 0.45947331190109253, -0.48364508152008057, 1.2121883630752563, -1.4498636722564697, 0.14999055862426758, 0.8672990798950195, -1.1491494178771973, 1.1521376371383667, -0.21865974366664886, -0.43494918942451477, -0.30916357040405273, 0.220859557390213, -2.702462911605835, -0.7143734097480774, -0.44023746252059937, 0.11527729779481888, -2.0408148765563965, 0.11320491880178452, -0.3486465513706207, 0.8922141790390015, -2.1069414615631104, -0.9290239810943604, -0.32456910610198975, 2.6765997409820557, 0.4337712526321411, 0.9245163798332214, -1.048622488975525, 0.5370441675186157, -0.903633177280426, 0.7678350806236267, -0.7945953607559204, -1.043484091758728, -0.21168160438537598, -0.8921329379081726, -0.7616819739341736, -0.9770781993865967, 2.0895113945007324, 1.2652459144592285, -0.2593897581100464, 1.256905198097229, -0.3588521480560303, 0.538811981678009], [2.5923800468444824, -1.3749563694000244, 0.9344758987426758, -0.18709543347358704, -1.0630398988723755, -0.02296248823404312, 0.6527633666992188, -1.0906291007995605, 0.13017961382865906, -0.8548334240913391, 0.5415372848510742, -1.5214958190917969, 0.3127163350582123, -1.3270457983016968, 0.26328885555267334, -0.666567325592041, 0.8798934817314148, 0.4422968029975891, 0.08890153467655182, 0.19286039471626282, 1.4087644815444946, -1.231956958770752, -0.15456482768058777, 2.1743273735046387, 0.3104774057865143, -0.48682960867881775, -3.187962770462036, -1.765368938446045, -0.23829784989356995, 1.3793846368789673, -0.8497158288955688, -0.15994761884212494, -0.03303859010338783, 0.23589874804019928, 0.6655268669128418, -0.8313676118850708, 2.1327521800994873, -0.146835058927536, -0.47211959958076477, 0.43101924657821655, -0.2595936357975006, -0.08789991587400436, -0.5708028674125671, -0.13398809731006622, 0.3171561062335968, -0.006724226754158735, -0.7195139527320862, -0.311492383480072, -0.6611205339431763, 1.0347481966018677, 1.1371700763702393, -0.3491807281970978, 0.7686344981193542, 1.3704642057418823, -0.11636777967214584, -1.4955593347549438, 0.9291454553604126, 1.2112025022506714, -0.5906493663787842, 0.11303379386663437, 1.0683741569519043, 0.5296849012374878, -1.8493636846542358, 0.5703964829444885, -0.9416604042053223, -0.7127828001976013, 0.1671663373708725, -0.48135751485824585, -1.168243169784546, 0.8766746520996094, -0.03472207486629486, 0.563520073890686, -0.05288954824209213, -1.0317095518112183, -0.641572892665863, 1.5108811855316162, 0.35201963782310486, 1.5942662954330444, 0.3883369565010071, -0.8688967823982239, -1.289434790611267, 0.24772043526172638, -0.046167004853487015, -0.5415194630622864, 1.2191828489303589, -0.5300965309143066, -0.1502440720796585, -0.30991607904434204, 0.2567187249660492, 0.17576514184474945, -0.003148952964693308, -0.5763475894927979, 1.0102205276489258, -0.6342540383338928, 2.0680007934570312, -2.670811414718628, 1.277531385421753, 0.8408563137054443, -0.6282926797866821, 0.3634767234325409, -2.27998948097229, 0.6135918498039246, 1.2374387979507446, 0.008300380781292915, 1.5976145267486572, -0.28866493701934814, -1.1511250734329224, 0.27240556478500366, -0.008486198261380196, -0.3222866952419281, -0.4812394678592682, -0.4195534586906433, 0.7740891575813293, 0.9883127212524414, 1.2176114320755005, 0.499525249004364, -0.2354455590248108, 0.6576739549636841, 0.1179906576871872, -1.2543796300888062, -0.5128276348114014, 0.7744321227073669, 1.1451504230499268, 0.47919410467147827, -1.508690357208252, 1.666235327720642, 0.6761928796768188, -0.28320977091789246, 1.9935128688812256, 1.442081332206726, -0.792224645614624, -0.6893378496170044, 0.24187321960926056, -1.0945426225662231, -1.2724699974060059, 1.357030987739563, 0.8337624073028564, 0.6479849219322205, -1.0099283456802368, 0.545866072177887, -0.30470559000968933, -0.16333845257759094, -0.3713752031326294, 0.39585816860198975, -0.9394799470901489, -1.174734115600586, -1.0148308277130127, -0.043756045401096344, -1.5226554870605469, 0.3523671627044678, 0.5159233212471008, -0.24695034325122833, 1.148176670074463, -0.11868184804916382, -0.2851030230522156, -0.5504829287528992, 0.06494377553462982, -0.2692568004131317, 1.9140193462371826, -0.16289746761322021, 2.0592262744903564, 0.6446418762207031, -0.24472792446613312, -0.7842041850090027, 1.0273386240005493, 2.1003832817077637, 1.6364836692810059, 0.8758989572525024, -0.13125407695770264, 0.27145448327064514, -0.08186008036136627, 0.5637602806091309, 1.4030812978744507, -0.3173142671585083, -0.351439505815506, 0.22247397899627686, -0.32325175404548645, 1.6569832563400269, -0.8286832571029663, 0.9148719906806946, -1.381441593170166, 1.0366941690444946, -1.5150738954544067, -0.3116021454334259, -0.6260063052177429, -1.202749490737915, -0.6958889961242676, -0.529461145401001, 1.2027455568313599, 1.752945065498352, 0.25411269068717957, 0.5823361873626709, -1.3574799299240112, 0.3072844445705414, -0.4112027883529663, -1.264108657836914, 0.3833885192871094, 1.6277297735214233, 1.2008224725723267, 0.1813962459564209, 0.259875625371933, 0.45420485734939575, 0.5427724123001099, -1.9973286390304565, 1.1678858995437622, 0.5374715924263, -0.1310626119375229, 2.1230907440185547, -0.8370160460472107, 0.13456226885318756, 0.7031672596931458, 0.002082561608403921, -0.8607997894287109, -1.5888334512710571, 0.3008672893047333, -0.0006763524725101888, -0.056947994977235794, 0.11541030555963516, 1.1675705909729004, 0.7977786660194397, -1.5330458879470825, -1.4609721899032593, -1.151925802230835, -0.3225230872631073, 2.4827098846435547, 0.05987444519996643, -1.1829228401184082, 1.544546365737915, -1.1348192691802979, 0.4741917550563812, 2.2636678218841553, -0.9639161229133606, 1.0176894664764404, 1.4016538858413696, -1.6847320795059204, -0.03072185255587101, 0.10082469880580902, 0.42207324504852295, 1.2469656467437744, 1.0437138080596924, 0.9999123811721802, 0.5357056260108948, 0.3148620128631592, -0.22377470135688782, 1.6943690776824951, 1.0344972610473633, -1.328995943069458, 0.644485592842102, 0.9678001403808594, 0.9960602521896362, 0.49717074632644653, 1.8454738855361938, 0.5554113984107971, 0.001569729414768517, -1.9514262676239014, 0.7588054537773132, 0.4486583173274994, 0.5108422636985779, -0.7661256790161133, 0.3022877275943756, -0.23102891445159912, 0.7229223847389221, -0.529028594493866, -0.1871393322944641, -1.5935332775115967, -0.21818886697292328, 0.9089786410331726, -1.5270107984542847, 0.7095922827720642, 1.5357779264450073, -0.1678512692451477, 0.9018080234527588, -0.7332346439361572, 1.275579571723938, 0.5362894535064697, 0.28204745054244995, -0.41224437952041626, 0.42414313554763794, 0.7507574558258057, 0.5006946921348572, 0.5549084544181824, 0.6486531496047974, -0.09543625265359879, -0.9365346431732178, 1.7066245079040527, -0.07093191146850586, -0.996596097946167, -0.9776955842971802, -0.5394763350486755, 1.1934775114059448, 3.4946506023406982, 0.53505939245224, 1.8101279735565186, 0.35165640711784363, -0.06036756932735443, 0.07877825945615768, -0.08156456798315048, 1.2533220052719116, 0.6968502402305603, 0.02778029814362526, -0.9715773463249207, 0.028301799669861794, -0.6075785160064697, 0.6638166308403015, -1.492864966392517, 0.07619144767522812, 0.130923330783844, 2.048384428024292, 0.5651275515556335, 1.1433494091033936, -1.4765061140060425, 1.0454192161560059, 0.08407731354236603, -1.0241340398788452, 1.588800311088562, -0.2997860908508301, 1.4573181867599487, -0.15599094331264496, -1.4325060844421387, 1.5183922052383423, 0.4927108585834503, 1.5625931024551392, 1.1644275188446045, 0.43527835607528687, 0.9514726996421814, 1.2186630964279175, -0.09118298441171646, 0.008756265044212341, -1.0165228843688965, -0.7415539622306824, 0.3219618499279022, 1.2043285369873047, -0.627411961555481, -0.010309486649930477, -1.0951166152954102, -0.009384023025631905, 1.0436617136001587, -1.1932920217514038, -0.32304227352142334, 0.27313414216041565, 1.9175498485565186, 1.1065610647201538, 1.2754794359207153, -1.8815399408340454, 1.810718297958374, -1.6340497732162476, 1.543561339378357, 1.8022488355636597, -1.300868034362793, 0.5531008243560791, -0.7055748105049133, -0.9473947882652283, -0.07096914201974869, 0.8074880838394165, -1.3103607892990112, -0.31022652983665466, -0.08516804873943329, 0.12309398502111435, 1.6619350910186768, -1.6600303649902344, 1.662726879119873, -1.698830246925354, -0.4118441045284271, -0.8263110518455505, 1.595261812210083, -0.33372020721435547, -1.308681845664978, -2.19929838180542, -0.4660682678222656, 0.12319407612085342, 0.293423056602478, 0.7381629943847656, -0.5698601603507996, -0.4378306269645691, -0.3950926959514618, 0.43918082118034363, -0.09053860604763031, -1.2119492292404175, -0.7034845948219299, 0.04544675350189209, 0.09798385947942734, 0.31547316908836365, 0.8800953030586243, 1.261458396911621, 0.11188303679227829, 0.7114781737327576, -0.02027078904211521, -3.468038558959961, -0.30486878752708435, -1.5515775680541992, 0.0513552762567997, 0.7873309850692749, 1.440101146697998, 0.7297855615615845, -0.5405543446540833, 0.10910187661647797, -0.056327685713768005, 0.570189893245697, -0.22593925893306732, 2.1934056282043457, 1.211090087890625, 0.051190074533224106, 2.022961139678955, -0.9644356369972229, 0.21169331669807434, -0.23722991347312927, -1.316068172454834, -2.9733598232269287, -0.7906384468078613, 0.37870097160339355, -0.6553057432174683, -0.6926734447479248, 0.5713557600975037, 0.28954577445983887, 0.08224283903837204, -0.9190639853477478, -1.2612700462341309, -1.272666096687317, 0.13091763854026794, -0.15310543775558472, 0.10345949232578278, 2.142437696456909, 0.8853222131729126, 1.5803991556167603, 0.4369719326496124, -1.5826431512832642, -0.9820528030395508, 1.5561769008636475, 1.3976298570632935, -1.4003657102584839, -0.4414147436618805, -0.004524562042206526, 1.4964237213134766, 0.24080415070056915, -0.388275146484375, 0.8988294005393982, 0.25720933079719543, -0.22672167420387268, -0.34285682439804077, 1.1737726926803589, -0.16169869899749756, -0.7223220467567444, 0.9018738865852356, -0.23113904893398285, -1.1121156215667725, -0.08842481672763824, 1.0239976644515991, 0.5570113658905029, -0.17154161632061005, 0.8014452457427979, -1.3913638591766357, 0.45651182532310486, -1.4818320274353027, -2.5845143795013428, 0.9817541837692261, 0.1066645160317421, 0.1985785961151123, -0.9436452984809875, -1.187872052192688, -0.13802534341812134, -0.5245307683944702, -0.42855513095855713, -1.6765795946121216, 1.4235817193984985, -1.9034863710403442, -0.9540225863456726, -0.8816007971763611, -0.32668548822402954, 0.07306763529777527, -1.0199460983276367, -0.9531042575836182, -1.105506420135498, 1.4418845176696777, 0.730905294418335, -0.4210219979286194, -0.3801617920398712, 1.0008807182312012, 0.8674086928367615, -0.7734173536300659, -1.7077209949493408, -0.9100539088249207, 0.29719802737236023, -0.4881768226623535, 0.21196909248828888, -1.014809250831604, 0.5038412809371948, -0.39423051476478577, 1.2279139757156372, -0.6644078493118286, 2.040438413619995, -0.1309848129749298, -0.26323139667510986, -0.3442436456680298, -0.845580518245697, -0.6721364855766296, 1.3705402612686157, 0.3228943943977356, 0.05237211659550667, -0.9705760478973389, -1.2022050619125366, -0.6505376100540161, 2.3705062866210938, -1.5069583654403687, -0.557679295539856, 1.4570345878601074, 0.24909420311450958, 1.9949004650115967, 0.6476579904556274, -0.5253889560699463, -0.07740234583616257, -0.5926612615585327, -1.6556071043014526, -0.9212914705276489, 0.047566767781972885, 0.6763744354248047, 1.787981390953064, 1.6124424934387207, -0.2803027331829071, -0.4183637797832489, -0.7733412981033325, 0.1885838657617569, 0.6665425300598145, -2.0687668323516846, -0.1115829199552536, -1.3384486436843872, 0.5828419923782349, -1.1356254816055298, -0.4848088324069977, 1.1208442449569702, 0.037959180772304535, 0.41959047317504883, 0.18543024361133575, 0.7327300906181335, 0.3404574692249298, 1.2605305910110474, 0.06490473449230194, -1.3290237188339233, -0.6741834282875061, -0.007325475104153156, 0.4548247754573822, -0.18229976296424866, -0.3667783737182617, 0.5972372889518738, -0.6717565059661865, 0.512146532535553, 0.6589891314506531, 1.986726999282837, -0.966696560382843, 0.13505804538726807, -0.2056313008069992, 1.196791648864746, 1.0039540529251099, 0.7915855646133423, -0.32060855627059937, 0.8155609965324402, 0.06948133558034897, -0.8509864211082458, 0.9705687165260315, 0.8947957754135132, -1.2342684268951416, 0.09865865856409073, -0.41010355949401855, -1.2649778127670288, -2.0665183067321777, 0.2515755593776703, -0.7134537100791931, 0.7295162677764893, -2.162651300430298, 0.5948423743247986, -0.5319758057594299, -2.1991500854492188, 1.8052858114242554, -0.8926092386245728, -1.8312163352966309, -0.8336690664291382, 0.2057252824306488, -2.035931348800659, 0.8440430760383606, 0.49992069602012634, 0.37632060050964355, -1.220902919769287, 0.6270684003829956, -0.6868192553520203, 0.471924364566803, 0.29176780581474304, -0.850315511226654, 0.4761205017566681, 0.6065792441368103, 0.36325550079345703, -0.6407837271690369, 1.232803463935852, 0.6063898801803589, 0.3336997926235199, -0.8524308204650879, 0.600442111492157, 2.140444040298462, 1.2770766019821167, -1.1551631689071655, 1.8885496854782104, 2.25028657913208, -0.2903939485549927, 0.7931337952613831, -1.2513257265090942, 1.4544272422790527, -2.0509932041168213, -0.022291073575615883, -0.20639416575431824, 0.09633656591176987, -0.5878369212150574, 0.27212440967559814, -0.03689916059374809, 0.005642198491841555, 0.6454104781150818, 0.0840543881058693, 0.9308072328567505, 1.0484541654586792, -0.8100917339324951, -0.06977158784866333, -1.6418060064315796, 1.2588833570480347, -1.8061327934265137, -0.4717216491699219, 1.571447730064392, -0.5928043723106384, 0.5097388625144958, -0.7153132557868958, -1.3785514831542969, 1.3413188457489014, -0.3866848647594452, 0.49644097685813904, 0.5512831807136536, 0.9749928712844849, 1.9192497730255127, -0.9883813261985779, 1.2858830690383911, -1.4441922903060913, -0.06696119904518127, -1.1917800903320312, -0.1546565443277359, -0.5451043844223022, 1.1090240478515625, -1.4772586822509766, -0.8130754828453064, 0.39248061180114746, -1.4503483772277832, 0.11711812764406204, -0.1498941034078598, 0.9003553986549377, 0.7576038241386414, -2.0230793952941895, -0.22990839183330536, -0.05752119421958923, -0.0772266685962677, -0.5383834838867188, 0.18997308611869812, 0.8084461092948914, 0.05343049764633179, 1.3873635530471802, 0.7593984603881836, -2.8069541454315186, -1.6849600076675415, 0.07307370007038116, -1.9008638858795166, -0.2457365244626999, 0.2612537443637848, -0.020705249160528183, 0.6722384691238403, 1.3294258117675781, -0.04696691408753395, 0.8826871514320374, 0.560211718082428, -0.7394554615020752, 0.5503230690956116, -0.01360259484499693, -0.8929343819618225, -0.03393121436238289, -2.3001320362091064, -0.09071563184261322, 0.462678998708725, -0.20113001763820648, -0.8191584348678589, -0.16655607521533966, 0.471150666475296, 1.120688557624817, 1.3837580680847168, 0.13707922399044037, -0.26023125648498535, -0.8683417439460754, -0.48241323232650757, 0.07550852000713348, 0.9286255240440369, -1.4644858837127686, -1.4338244199752808, 1.3134593963623047, -1.3868783712387085, -1.1781482696533203, -1.7921857833862305, -0.5934939384460449, 0.7705382108688354, -1.7163587808609009, -1.598328948020935, 0.4281310439109802, 0.11797314137220383, -0.08759639412164688, 0.5146921873092651, 0.7449772953987122, -0.5476056337356567, 0.8232133388519287, 2.02731990814209, -0.2325689047574997, -0.10590792447328568, 0.10137380659580231, 1.1725573539733887, 0.07596519589424133, -0.2753894329071045, 0.0789734274148941, 0.9423127174377441, -0.693747878074646, 0.3393349051475525, 1.074238657951355, 0.4680198132991791, -1.2004780769348145, 0.332417756319046, -0.2371709793806076, -0.7670384645462036, -0.06944859772920609, -0.982475757598877, 0.536040186882019, 0.5439884066581726, 1.830599308013916, 0.38544297218322754, -0.5353984832763672, -0.6186572313308716, -0.4413433372974396, -0.25250065326690674, -0.5336283445358276, 1.5718694925308228, -0.334758460521698, 0.1462438404560089, 0.7630048394203186, 0.29354703426361084, 1.007377028465271, -1.1781201362609863, 0.8939690589904785, 0.05876545608043671, -1.092406988143921, -1.2706222534179688, 0.18121162056922913, -0.045249130576848984, -0.6767616271972656, -0.0229971781373024, -0.36501985788345337, -0.9990270137786865, 0.2479596734046936, 0.308830589056015, 1.0138676166534424, 0.3787974417209625, -0.7211858630180359, 0.6605058312416077, -0.147868350148201, 0.01080489344894886, 1.1038250923156738, -0.3145074248313904, -1.0092997550964355, -0.1681946963071823, -2.16292667388916, 0.800851047039032, 0.9254676699638367, -0.11345551908016205, 0.9165960550308228, 0.6209741830825806, -0.33959072828292847, -0.869324803352356, -0.8596843481063843, -1.109313726425171, 1.1374551057815552, -0.9014293551445007, 2.467308759689331, 0.6199209690093994, -1.5016990900039673, -0.7237754464149475, 1.8936381340026855, -2.4065544605255127, 0.845650851726532, 1.0503687858581543, 0.5388128757476807, 0.4602617621421814, -0.6804506182670593, 1.4423125982284546, -0.4670083522796631, 1.7392996549606323, -1.7354954481124878, 1.3211277723312378, 0.5981728434562683, -0.3141268789768219, -0.9599389433860779, -1.4681934118270874, 1.1141133308410645, -0.5599762201309204, 0.332518607378006, 0.7450183033943176, 1.2005704641342163, -0.1694430410861969, -0.6688550114631653, -0.295487642288208, -1.0610976219177246, 1.4277323484420776, -2.4995603561401367, -0.25302043557167053, 1.0398321151733398, -0.735743522644043, -1.0146827697753906, -0.389861136674881, -1.0363770723342896, 0.2896861732006073, -0.06996134668588638, -0.788623034954071, 1.598180890083313, -0.743062436580658, 1.2462694644927979, -0.7123522162437439, -0.9808827042579651, 0.054109591990709305, -0.1730830818414688, 0.6353399753570557, 0.6387806534767151, 1.7358338832855225, 0.16788823902606964, 1.7660610675811768, -1.1568653583526611, -0.9255058169364929, 0.6655332446098328, 0.02879863977432251, 1.1238460540771484, 1.197926640510559, -0.9492834210395813, 0.8120445013046265, 1.1579235792160034, 0.1279415786266327, 1.412238359451294, -0.46776241064071655, 0.9577181339263916, -0.273529589176178, -1.4693207740783691, -1.2977982759475708, -0.9285280704498291, 0.06079933047294617, 0.7874975800514221, -1.6629818677902222, -0.3838382065296173, -1.6507790088653564, -0.9290273785591125, -0.6904395818710327, -1.6210689544677734, -0.9820207357406616, 0.7976552844047546, 0.54302579164505, 0.5089070200920105, -0.10503532737493515, 1.0899219512939453, -0.05464301258325577, 1.4799842834472656, -0.3747955858707428, 0.6777366995811462, 0.9965563416481018, -2.704742670059204, -1.177008032798767, -0.26439616084098816, -0.3317437171936035, 2.1453778743743896, -0.6734517216682434, 0.7820392847061157, -2.04580020904541, -0.13315068185329437, 1.1875101327896118, -0.3729599714279175, -0.6138800382614136, 2.8865933418273926, -0.5604645609855652, -1.3020678758621216, -0.40364763140678406, -1.3665138483047485, -0.9372736215591431, 0.05865660682320595, 0.7507006525993347, -0.9394395351409912, -1.0370826721191406, -0.08073606342077255, -0.2674911320209503, -0.3063538074493408, 0.06223529949784279, -0.5690236687660217, -0.7875487804412842, -2.562448740005493, -0.10376514494419098, -1.5597715377807617, -0.41905850172042847, -2.1259374618530273, 0.33806443214416504, 1.135801076889038, 0.7362615466117859, 0.6608629822731018, 0.10904212296009064, -0.8520758748054504, 0.007014111615717411, -0.8256761431694031, -0.4094391465187073, -0.986202597618103, -1.1451038122177124, 0.723300039768219, -1.4601205587387085, -0.9788994789123535, 1.3501752614974976, -1.4347363710403442, 0.22227662801742554, 0.27359339594841003, 0.2650516629219055, -1.296413779258728, -0.1466943621635437, 0.5519750714302063, 1.0301759243011475, 0.930753231048584, -1.101014494895935, 0.0634046345949173, -0.279783695936203, -0.9775980710983276, 0.7537757754325867, 1.8148952722549438, 0.2714917063713074, -1.070072889328003, -0.6915317177772522, -0.978264570236206, 1.5071043968200684, 0.7814989686012268, 0.9805504083633423, -0.30498337745666504, -1.8290334939956665, -0.6756404042243958, -1.333055019378662, 1.4382892847061157, -0.2237454056739807, -0.20032554864883423, 1.7457045316696167, -0.7651692032814026, -0.1526416838169098, 0.830657422542572, 0.6776089072227478, 1.8212136030197144, -0.3514402508735657, -0.1862529069185257, -0.9102935194969177, -1.4585374593734741, -0.5389545559883118, -0.039542071521282196, -1.5753971338272095, -1.1514452695846558, 0.8446236252784729, -0.42050737142562866, 0.2663593590259552, 0.3760035037994385, 0.796600341796875, -0.44552209973335266, -0.4164203703403473, 0.8390347361564636, -1.6342756748199463, -1.806403398513794, -0.3770667612552643, -0.40553194284439087, 0.8982739448547363, -1.03336501121521, 2.0142409801483154, 0.2962382733821869, -0.6200266480445862, 0.7288007140159607, 0.21670497953891754, -0.044197529554367065, 1.1248828172683716, 0.9780140519142151, -0.3926933705806732, 0.8319947123527527, -1.195420503616333, -1.064690113067627, 0.5204137563705444, -1.3642746210098267, -0.9021992087364197, -1.3066260814666748, -0.6760596632957458, -0.5981118679046631, 0.9860333204269409, -1.5805054903030396, -0.4166029095649719, -0.8224331736564636, -1.462557077407837, 0.7355530858039856, -1.210422158241272, 0.7071377038955688, 0.10777577012777328, 1.2252895832061768, -0.6845583915710449, -1.1860345602035522, -0.26029741764068604, -1.3833863735198975, -0.4478284418582916, -1.077057123184204, -1.2920355796813965, 1.802109956741333, -0.007302206475287676, -1.2328779697418213, 0.20001855492591858, 0.3206811845302582, -0.15253037214279175, -2.010239601135254, 0.3016672432422638, -0.06321176886558533, -0.9077699780464172, 0.2833649516105652, 0.4278498589992523, -0.8332188725471497, 0.8123573660850525, 0.13865137100219727, 0.8183039426803589, -0.9090002775192261, 0.8655644655227661, 0.6025357246398926], [-1.043823003768921, -0.8578619360923767, 0.16598935425281525, 0.5624790191650391, 1.0082021951675415, -1.092990517616272, -0.3625449538230896, 1.7042548656463623, 0.8208510875701904, -1.3824189901351929, -2.962329864501953, 1.2543355226516724, 0.23981136083602905, 0.1440022736787796, 0.45300719141960144, -1.663277506828308, -0.6870927810668945, -1.6348131895065308, 0.7684010863304138, -0.42831283807754517, -0.7743425965309143, -0.4793570041656494, -0.6066799759864807, 0.4500868320465088, 0.9327517747879028, 0.17826752364635468, 1.6890220642089844, -0.6892151236534119, 0.30466312170028687, -0.2428576797246933, -0.907068133354187, 0.6446701884269714, 1.2340914011001587, -1.7490493059158325, 1.1930862665176392, 0.35527274012565613, 2.124368190765381, -0.0037306149024516344, 1.6650105714797974, 1.5134572982788086, 0.48124030232429504, 0.28161048889160156, 0.6631896495819092, 0.02903398685157299, -0.19528327882289886, -0.440396785736084, -0.6133326292037964, -0.3293747305870056, 0.7180091738700867, -0.26357731223106384, 1.9288846254348755, -0.5132592916488647, -1.6093119382858276, 1.4200525283813477, -0.9863373637199402, -1.0868436098098755, 1.5614675283432007, 1.8238052129745483, -1.7792476415634155, 0.418022483587265, -1.692217469215393, -0.3416948616504669, 0.952515721321106, 0.6931641101837158, -0.021062618121504784, 0.26290085911750793, 0.024402478709816933, 0.8815731406211853, 0.30644214153289795, 0.018022045493125916, -1.0312602519989014, 0.13394401967525482, -1.0144670009613037, -0.47749271988868713, -0.28931108117103577, -0.8840306997299194, -0.8266805410385132, -0.6380465030670166, -0.7931167483329773, 1.4650616645812988, -0.8639101982116699, 0.1941358745098114, -0.5651704668998718, 0.4858664870262146, 0.9917475581169128, 2.5164737701416016, -0.9867599606513977, -0.5583824515342712, -0.550642728805542, -0.9538276195526123, -1.767122745513916, -1.5387656688690186, 0.9387653470039368, -2.498323440551758, 0.2803073823451996, 1.4053410291671753, -0.8267046809196472, -0.046020954847335815, 1.349489450454712, -0.5029616355895996, -0.35372868180274963, 0.5204747319221497, 0.414824515581131, -0.6873878240585327, 0.034986477345228195, 2.2064521312713623, 0.2510717809200287, 0.16020861268043518, 0.6111147403717041, 0.384203165769577, 0.5801916718482971, -2.459216594696045, 0.2481534332036972, 1.1167631149291992, -0.07579080015420914, -1.5439860820770264, 1.1912223100662231, -0.09818282723426819, -1.2960351705551147, 1.377116084098816, -1.0064935684204102, -0.38255617022514343, 0.9300424456596375, 0.9173768758773804, -0.47031548619270325, 2.0304229259490967, 0.768285870552063, 0.42419153451919556, 0.5499133467674255, -0.3371374011039734, 1.1329174041748047, 0.0854014977812767, 1.847285270690918, 0.657320499420166, -0.6695184111595154, 0.12348870187997818, 1.29461669921875, 2.2592544555664062, 0.23926062881946564, 0.7051340341567993, 0.2779329717159271, -0.2733560800552368, 0.07909759134054184, 0.465402215719223, 2.2534260749816895, -0.21456487476825714, 0.11939756572246552, 0.5367549657821655, -0.19712673127651215, 0.05493544042110443, -1.2985392808914185, 0.3169681429862976, 0.5517438054084778, -0.3882865011692047, 1.6171865463256836, 0.48532119393348694, 0.3531136214733124, 0.06353010982275009, 0.38249966502189636, -0.4778256118297577, -1.1012552976608276, -0.6228330731391907, -0.2654743194580078, -1.816856026649475, 0.6253527998924255, 0.4071531891822815, -1.7303521633148193, 0.416195273399353, 0.4712851941585541, -2.781588315963745, 1.2974464893341064, 1.4438999891281128, -1.852285623550415, -0.27427953481674194, 0.7233478426933289, -1.0143089294433594, -0.5771241784095764, 0.0745038092136383, 0.9131794571876526, 1.3993985652923584, 0.47063198685646057, 1.7297717332839966, -0.44396698474884033, -1.100415825843811, -1.5475274324417114, 1.7781723737716675, -1.7602094411849976, -0.4948473274707794, -0.2137218564748764, 1.802302598953247, -1.2941768169403076, -1.0901786088943481, -1.3225208520889282, 0.5507552027702332, -2.0195486545562744, -0.8885777592658997, -1.0688824653625488, 0.9086460471153259, 0.39632126688957214, -1.2702034711837769, 0.06470450758934021, -1.287638545036316, -1.3961076736450195, -0.5220535397529602, 0.2662598192691803, -1.2894129753112793, 1.6046690940856934, -0.4010147154331207, -0.37085598707199097, 0.3635968863964081, 0.47489282488822937, 1.426877498626709, 0.263264000415802, 0.9225441813468933, 0.20794689655303955, 1.0985164642333984, -0.9117991924285889, -1.2771178483963013, 0.28835704922676086, 0.21736855804920197, 1.9345864057540894, 0.6922793388366699, 1.9339267015457153, 0.4117540717124939, -0.6972458958625793, -0.4950665235519409, 0.9430955052375793, -0.10586531460285187, 1.6675257682800293, -0.4364789128303528, 0.39161205291748047, 1.9920815229415894, -0.9977395534515381, 0.12012159079313278, -0.12937338650226593, -0.2908235788345337, 0.07720988988876343, -0.041882969439029694, 0.47975069284439087, 0.3001317083835602, 1.7060706615447998, 0.5937652587890625, -1.139229655265808, 0.9712892174720764, 0.9069516658782959, 1.8611098527908325, -0.5143001079559326, 0.6810877919197083, -0.9197685718536377, -0.454706072807312, -0.3024504780769348, 0.8764098882675171, 0.12243739515542984, -0.9624044299125671, 0.7416645288467407, -0.24340108036994934, -0.021699732169508934, 0.024631522595882416, 0.8323904871940613, 0.23338085412979126, -1.5316004753112793, -0.30044689774513245, -0.5240998268127441, 1.2850451469421387, 0.48050054907798767, -3.058182716369629, -0.45475152134895325, -0.4805077016353607, 0.537991464138031, 0.24445272982120514, 1.2116918563842773, 1.6713591814041138, -1.9769450426101685, -0.45599234104156494, 0.4921742081642151, -1.0563874244689941, 0.20631219446659088, -0.9417185187339783, 1.5618830919265747, -0.6309621930122375, 0.3210459053516388, 0.4352194666862488, 0.42043542861938477, -0.18273113667964935, -1.3174761533737183, -0.2921656370162964, 0.1910068839788437, 0.8142632246017456, -0.5168738961219788, -0.3852025270462036, 0.047057997435331345, 0.2398713231086731, -1.5642536878585815, -0.7353962063789368, 0.11206451803445816, 0.021081959828734398, -1.651353120803833, -0.8961330056190491, -0.8708507418632507, 0.41959524154663086, -0.25863102078437805, 0.1722421646118164, -1.609692096710205, 1.883601188659668, 0.1205931082367897, 0.4833410680294037, -1.1052374839782715, -0.4067222476005554, 1.3921416997909546, -0.6060031056404114, -1.2753760814666748, 0.08361101150512695, -2.0693821907043457, -0.07091744244098663, 0.3573516011238098, -2.8761837482452393, 0.1580996811389923, -0.8756964802742004, -0.6353772282600403, -0.19516566395759583, -1.183066725730896, -0.9718654155731201, -0.8363515734672546, -0.7368453145027161, -0.5838128328323364, -0.3421550691127777, -0.1493063122034073, 0.4229700565338135, 0.5503592491149902, -1.0144814252853394, 0.186613067984581, 0.8084377646446228, -0.5399494767189026, 0.24470454454421997, 1.7515789270401, 0.5912048816680908, -0.7808972001075745, 0.7929818630218506, 1.5765851736068726, 1.2899123430252075, -0.6247789263725281, -0.3424804210662842, 0.1943681389093399, -0.42130154371261597, -0.45912328362464905, -0.7003254294395447, -1.342033863067627, 0.1528184860944748, 0.6087345480918884, -1.9669920206069946, -1.48777437210083, 0.45316311717033386, -0.3624982237815857, 0.8231685161590576, -0.599799633026123, 1.8193142414093018, -0.5415529012680054, 0.14315272867679596, 0.40550604462623596, -1.0457221269607544, -0.4420674443244934, 0.6074892282485962, -0.10596692562103271, -1.0857146978378296, 0.38623300194740295, -0.5362881422042847, -0.28376010060310364, -0.836308479309082, -1.8321315050125122, -1.0493924617767334, -0.8855119943618774, -1.1169731616973877, 0.01747715286910534, 2.439113140106201, -1.0069541931152344, 0.194259911775589, 1.6144659519195557, 1.8798421621322632, -1.1532472372055054, 0.08538409322500229, -0.44368264079093933, 0.8964532017707825, -1.7421283721923828, 1.08266019821167, -0.9086177945137024, -0.9378272891044617, 1.8135955333709717, -1.2780780792236328, -0.5180086493492126, 0.8763633370399475, 0.5308694839477539, -0.30468764901161194, -0.7114899754524231, 0.46646928787231445, -2.101543664932251, 0.6107635498046875, -0.6265320777893066, 0.248148575425148, -1.3895028829574585, 1.1877557039260864, -1.8548011779785156, 0.08547559380531311, -1.1148338317871094, -0.927161693572998, 0.5037559866905212, -0.6170437335968018, -0.05171018838882446, 2.573967218399048, 0.6007339358329773, 0.5638154149055481, 1.3203049898147583, 0.1318006068468094, 0.19065196812152863, 1.144507646560669, 0.8500903844833374, 0.5784386396408081, -0.03968586027622223, 0.3556041717529297, 0.5931525230407715, -0.6367034316062927, 0.8799276947975159, -0.1100117564201355, -0.5445899963378906, -0.11694371700286865, -1.7483588457107544, 1.1804065704345703, 1.1119340658187866, -1.2594263553619385, 1.1071579456329346, 0.5165227651596069, -1.7520617246627808, -0.0697951465845108, -0.20164287090301514, -0.7457727789878845, 0.33987095952033997, -0.2600290775299072, -1.268433928489685, 0.13146571815013885, 1.3117645978927612, -0.73060542345047, -0.7201211452484131, 1.3834116458892822, 0.2207595854997635, -0.8049035668373108, 1.7995237112045288, 0.8188639283180237, 0.7248347401618958, 0.404431015253067, -0.28453412652015686, -0.15555228292942047, -0.15593193471431732, -1.019612193107605, 2.0724680423736572, 0.4316413104534149, 0.2596939206123352, -1.28196120262146, -0.10880250483751297, 0.17293192446231842, -0.6029279232025146, -1.7142962217330933, -0.30996471643447876, 0.8642776012420654, -2.3644824028015137, -0.16473603248596191, 0.09098585695028305, -1.5162392854690552, -1.3411905765533447, 1.2899935245513916, 0.12451782822608948, 0.5536585450172424, 0.5442354083061218, -0.24446024000644684, -0.587470531463623, -0.8471566438674927, 0.17245694994926453, -3.1659224033355713, 2.8078160285949707, 0.6244631409645081, 0.8166283965110779, -0.27331826090812683, 0.8490474820137024, 0.08619417995214462, 0.031386736780405045, -0.3463139235973358, 0.32695525884628296, -1.2258028984069824, 0.09695737063884735, 2.6004369258880615, 0.44391319155693054, 0.3361045718193054, 1.783846378326416, 0.12233128398656845, -0.17938855290412903, -0.06643160432577133, 1.8795807361602783, -1.5367611646652222, -0.5578012466430664, -0.49374905228614807, -2.4702422618865967, 0.7341585755348206, -0.37761011719703674, 1.4271306991577148, -0.29073458909988403, -0.20343025028705597, -0.6091985702514648, 0.05130257084965706, -0.13254663348197937, 0.192281112074852, 0.46096402406692505, 0.06851700693368912, 0.21304979920387268, -0.490739643573761, -0.9080754518508911, -1.7968053817749023, 1.684136986732483, -0.30579376220703125, -1.4206870794296265, 0.9306418299674988, -0.07041636854410172, 1.9330400228500366, 1.5440844297409058, -0.6103039979934692, -1.9711672067642212, -1.0854910612106323, 1.1823582649230957, 0.8247197866439819, -1.566422939300537, 1.8298225402832031, 0.08070272207260132, 1.6279603242874146, 0.3308345377445221, 0.5708613991737366, 0.13500873744487762, -0.617560863494873, -1.3916500806808472, -0.8357275724411011, -1.1998236179351807, -0.4603891372680664, -0.3907233774662018, -0.11621939390897751, 0.3128810226917267, 0.5247644782066345, 1.1126439571380615, 0.4594174027442932, 0.7285962104797363, 0.3355182111263275, 1.3356847763061523, 1.9875177145004272, -0.8197864890098572, -0.2865767180919647, -0.4461933374404907, -0.16108661890029907, -0.49954545497894287, -0.5891388654708862, 0.6033798456192017, 0.2375848889350891, 0.441796213388443, -1.874738097190857, 1.179467797279358, 0.5152917504310608, -1.2790786027908325, -1.9840296506881714, -0.09298017621040344, 0.31744033098220825, -0.8187986612319946, 0.4805011451244354, -0.06966062635183334, -0.596835196018219, -3.3615384101867676, -1.2034138441085815, 0.8644994497299194, -0.8448962569236755, 0.9138603210449219, 0.5980663299560547, 0.5101108551025391, -0.24011263251304626, 0.33411911129951477, -1.2794753313064575, 0.1549360603094101, 0.6759252548217773, 0.023401791229844093, -0.7787138819694519, -3.560654401779175, -1.3380283117294312, 1.902155876159668, 0.8294165134429932, 1.9489797353744507, -1.0903254747390747, 0.13478867709636688, 1.6491833925247192, 1.601401686668396, -2.6492209434509277, -0.006720955017954111, 0.7982362508773804, -0.6109170317649841, -1.0230227708816528, -1.7507264614105225, 1.7710449695587158, -1.100488543510437, 0.9716318249702454, -0.2851788103580475, 0.21783868968486786, 0.7941673994064331, 0.1794079691171646, -0.9335831999778748, 0.6252204179763794, -0.5619126558303833, 2.6861929893493652, -0.7661337852478027, 0.3703237473964691, -0.6053480505943298, -0.11632110923528671, 0.9515013098716736, 1.0399144887924194, 0.7674216628074646, 0.528327465057373, 0.5195599794387817, 2.5110180377960205, 0.7212314009666443, 1.4010387659072876, -1.4886479377746582, 0.98085618019104, 0.6169141530990601, 0.9540688395500183, 1.2818249464035034, 1.295849323272705, 0.5694023370742798, -2.5705509185791016, 1.9090845584869385, -1.8830779790878296, -0.5140059590339661, -1.411009669303894, -0.5703914761543274, 0.48388299345970154, 0.10280945897102356, -1.9260780811309814, 1.3356562852859497, -0.9651995301246643, -0.6545634269714355, 0.20718100666999817, 1.3306423425674438, 1.8711674213409424, 1.6989936828613281, -2.7862319946289062, -1.3070858716964722, 1.1509908437728882, -1.0825554132461548, 0.34111595153808594, -1.3316307067871094, 0.7655509114265442, 1.5924797058105469, 0.6732280254364014, 1.9224984645843506, -1.2587910890579224, -0.40160128474235535, 0.5039176940917969, -0.017503490671515465, 0.34523019194602966, -0.5298221111297607, -1.2124049663543701, -0.8746438026428223, 0.06467536091804504, 1.3679742813110352, -0.8016547560691833, -0.22006693482398987, -0.0647372156381607, 0.9950544238090515, -0.7898467779159546, 0.21618478000164032, -0.6413907408714294, 1.526774287223816, 1.2258784770965576, 0.14113987982273102, -0.820176362991333, -0.6064367890357971, -1.0767992734909058, 0.4002319574356079, -0.841630220413208, -1.2961088418960571, -0.8880490064620972, 0.15766583383083344, -0.6282694935798645, 0.2624114453792572, 1.172722339630127, -0.023912541568279266, -2.1170408725738525, 0.958667516708374, -0.7622743844985962, 0.8995459079742432, -1.535035490989685, -0.36776819825172424, -2.0605077743530273, 1.6343351602554321, -0.6866587996482849, 0.9153929352760315, -0.7481292486190796, 0.4361380934715271, 0.5479157567024231, 3.005070209503174, -0.8372976779937744, -0.09997402876615524, 0.4454182982444763, -0.704751193523407, -0.6191748380661011, -1.2335453033447266, -1.126867651939392, 0.9682819247245789, 1.3996385335922241, -0.899084746837616, -1.5351697206497192, 0.5876684784889221, -0.7629262208938599, 1.1377947330474854, -1.765397071838379, -1.0050113201141357, -1.8692748546600342, -0.3688286542892456, 1.6453478336334229, 1.373274326324463, -0.10656467825174332, -0.8451784253120422, -0.5738955140113831, -1.195570468902588, 0.5715954303741455, -0.09834690392017365, 0.24395431578159332, -1.895486831665039, 1.0151402950286865, -0.07481759786605835, -0.20471540093421936, -0.3321855366230011, 0.7604368329048157, -0.9484758973121643, -1.8260478973388672, -1.15057373046875, -0.1726948469877243, 0.4876825213432312, -0.8230799436569214, 0.6553989052772522, 0.18362125754356384, -1.1628283262252808, 0.6761776804924011, 0.5207119584083557, 0.9995757937431335, -0.6013952493667603, 0.08818564563989639, 0.03873607888817787, 1.4896016120910645, -0.2590823471546173, -0.6693809628486633, 0.7733636498451233, 1.3566983938217163, 0.7226796746253967, -1.5149140357971191, 1.5342721939086914, -0.6021702289581299, 0.32827305793762207, -0.6467259526252747, -1.3236989974975586, -0.5288591384887695, 0.538043200969696, -1.0279433727264404, 0.3059287369251251, 0.1643170267343521, 1.2845178842544556, -1.5922260284423828, 0.3803025186061859, 1.7235755920410156, -0.7541940808296204, -0.26266077160835266, -0.2140156775712967, -0.05429422855377197, 0.23105308413505554, -0.7521286010742188, 0.1762939989566803, -1.393394947052002, 0.9845938086509705, -0.10034452378749847, 0.03163756802678108, -1.0650153160095215, -1.3919605016708374, -0.9257449507713318, 0.8184058666229248, 0.6571944355964661, -0.1320585161447525, -0.801472008228302, 0.5767963528633118, -2.0893495082855225, -0.5119777321815491, 0.5670308470726013, -0.48965775966644287, 0.47888436913490295, 0.5459645986557007, -1.8458935022354126, -0.3618873059749603, -0.3272087275981903, -1.084639549255371, -0.9726063013076782, -1.0508501529693604, -0.2915080189704895, 0.4804483652114868, -1.6187363862991333, 0.6092858910560608, 0.5196672677993774, 0.5990956425666809, 2.2389228343963623, -0.6047057509422302, 1.217894196510315, 0.1967170685529709, 0.2908841669559479, 0.6181595325469971, -1.357347011566162, 0.16569602489471436, -0.742773711681366, 1.0612009763717651, -0.5862116813659668, -1.43711256980896, -0.7038251757621765, 0.15122196078300476, 0.7378873229026794, 0.7043390274047852, 1.719160795211792, 0.583993673324585, 0.8767589330673218, -0.5342167615890503, -0.4058257043361664, 0.5033244490623474, 0.0853644385933876, -0.4219305217266083, -0.9926615953445435, -1.4421249628067017, 0.7931883931159973, -1.5414600372314453, -1.2244411706924438, 0.4063239097595215, -0.16186633706092834, 0.766468346118927, -0.5220540165901184, -1.4421226978302002, -0.5469121336936951, -1.921350121498108, 0.6578474044799805, 0.7559268474578857, -0.047530706971883774, 1.1698287725448608, -0.5838453769683838, 1.437512993812561, -0.6427525877952576, -0.42928189039230347, -1.5494558811187744, 1.435699701309204, 1.174534797668457, 0.31302616000175476, 1.1575616598129272, -0.5617996454238892, -1.2427881956100464, 2.14658522605896, 0.12298785895109177, -2.2389321327209473, -0.6405612826347351, -0.4448142349720001, 1.8010404109954834, -1.5212562084197998, 0.4308724105358124, 0.7570409178733826, -0.767113447189331, 0.9839827418327332, 0.9473482966423035, 1.037434697151184, -0.24119310081005096, 0.4644233286380768, -0.7085597515106201, -1.0039498805999756, 1.0378755331039429, -1.0477708578109741, -0.7779367566108704, 1.3576877117156982, 0.5901052951812744, 0.36332154273986816, -1.5230064392089844, -0.5348202586174011, -0.6905214190483093, -1.1192408800125122, 0.6269717216491699, -0.382732093334198, 0.16649940609931946, -2.0781924724578857, -0.6926450729370117, 0.07565546035766602, -1.0135657787322998, 0.6145359873771667, -0.9698513150215149, -0.5286502242088318, 0.00907265767455101, -0.02498895861208439, -1.3997379541397095, 3.6814510822296143, 0.3900081217288971, -0.5606070756912231, -0.5429463982582092, -2.1630537509918213, -2.9496073722839355, -0.426163911819458, 0.6189060807228088, -1.0228333473205566, 0.49922141432762146, 0.5211414694786072, 2.684112071990967, -0.6474606990814209, -1.9509750604629517, -1.4677478075027466, 0.3208925426006317, -1.549433708190918, 0.4664437770843506, -0.7692656517028809, 0.15779650211334229, -1.6391124725341797, 0.38934648036956787, 1.1597929000854492, 0.9879540205001831, -1.4384573698043823, 0.6581631898880005, -1.3535128831863403, -0.32374557852745056, 1.399779200553894, -0.8081967234611511, 0.02619636431336403, -0.21189022064208984, -1.7887992858886719, -0.24300621449947357, 1.1162742376327515, -1.1449066400527954, 0.19707754254341125, 0.3333985209465027, 0.004617882892489433, 1.7123140096664429, 0.00478181429207325, -0.48517775535583496, 0.9924538135528564, -0.5457402467727661, 1.1921477317810059, 0.7959887385368347, -0.36413562297821045, 1.0501662492752075, 1.3036071062088013, -0.06431843340396881, 0.359811931848526, -0.30425888299942017, 1.8882988691329956, 0.6367133259773254, -0.426949143409729, -0.4125102162361145, 0.3354055881500244, -0.09269505739212036, 2.0790514945983887, 0.12321615219116211, -0.3518553674221039, -1.146528959274292, -0.49509361386299133, 1.798161268234253, -1.97187340259552, 0.0035611421335488558, 1.1380527019500732, -0.7156240940093994, 0.8549041748046875, -0.9222581386566162, 0.2806769609451294, -0.7647404074668884, 0.6844698190689087, -1.1743637323379517, 0.08980058878660202, -1.5782060623168945, -1.5354400873184204, -0.4882030785083771, -1.225555419921875, 0.13641245663166046, 0.4256531894207001, 0.1181206926703453, -0.9446724653244019, -0.977418839931488, -0.6370551586151123, -0.3706083297729492, 0.5277154445648193, -0.6930542588233948, -0.08062723278999329, -1.1027997732162476, -2.0448267459869385, -0.21961785852909088, 1.3599128723144531, -0.6061139106750488, -0.3799031972885132, -0.9577118754386902, -1.2790364027023315, -0.8896413445472717, -0.9839456081390381, -1.4252694845199585, -0.6863517761230469, -0.3884226083755493, 0.40913844108581543, 0.20975452661514282, -0.5901494026184082, 0.8812493681907654, 1.6688711643218994, 1.9476029872894287, 0.560080885887146, -0.5265596508979797, 0.49866461753845215, -0.3722112774848938, 0.10963805764913559, -0.2399010807275772, 0.725912868976593, -1.1402469873428345, -1.4883939027786255, -1.2334237098693848, -0.25562530755996704, 0.7948635816574097, 0.7791998982429504, 2.7372069358825684, 0.08521916717290878, -0.3794640600681305, -2.0033962726593018, -0.5530031323432922, 1.566413164138794, 0.06637389212846756], [-1.0926542282104492, 0.8091188669204712, 0.09803660213947296, -0.4118359684944153, 1.2044563293457031, -0.398958295583725, -0.4740164875984192, 0.1008303165435791, -1.0445185899734497, -0.9313396215438843, 0.8663251399993896, -0.08079733699560165, -1.302226185798645, -1.4398713111877441, 2.273256301879883, -0.7687011361122131, 0.15980859100818634, -0.6390158534049988, 1.0988245010375977, -0.004673198331147432, 0.1978341042995453, 0.232942596077919, 1.3093898296356201, -0.29481256008148193, 0.15034842491149902, -1.1331112384796143, 0.28765252232551575, 0.6899256706237793, 0.9945802092552185, 1.0013233423233032, -2.6236231327056885, 0.4924390912055969, -0.9867905974388123, 0.46459364891052246, -0.32947516441345215, 0.1652846783399582, 0.5008364319801331, 0.26485762000083923, -0.863739550113678, -0.09048628062009811, 1.2349311113357544, 0.5162087082862854, -0.46772500872612, 0.5360332131385803, -1.1119362115859985, 0.4155518710613251, 1.107765793800354, 0.8972812294960022, 0.2677202820777893, -0.5817065834999084, -1.0732535123825073, -0.30981406569480896, -0.5760679841041565, 0.5103267431259155, -0.7265231013298035, -1.6283347606658936, 0.45221367478370667, -0.4137594997882843, -0.3474137783050537, -0.19798453152179718, -0.4156985580921173, -0.19427934288978577, -0.1904730647802353, 0.747839093208313, -0.821410596370697, -1.6625311374664307, -0.8051868677139282, -0.4309260845184326, 0.05175270512700081, 0.9790929555892944, 2.054487705230713, 2.272278070449829, -1.8625160455703735, -0.5624313354492188, -0.47721484303474426, -1.3129349946975708, -2.0070207118988037, -0.4506520926952362, 0.7409764528274536, -2.010802984237671, -0.13219337165355682, 0.779870331287384, -2.2420990467071533, -0.07455898821353912, -0.524315595626831, 1.593443512916565, 0.140037402510643, 0.00048130584764294326, 1.0667740106582642, -0.13432130217552185, -2.1620945930480957, 1.2877304553985596, -0.6315619945526123, -1.13216233253479, -0.6235942244529724, 1.464536190032959, -2.0316781997680664, -0.6998029351234436, -0.39922747015953064, 0.6453437805175781, -0.9028577208518982, -0.9575966596603394, -1.4998843669891357, -1.0886906385421753, 0.727805495262146, -0.3242607116699219, 0.31101012229919434, -1.8264241218566895, -1.2632794380187988, 0.218521386384964, -1.0234606266021729, -1.5726712942123413, -0.24404184520244598, 0.4610327184200287, -0.4596192538738251, -0.09606055170297623, 0.30949586629867554, -0.3673105537891388, -0.1816137582063675, -0.792335569858551, 0.9704692363739014, 0.9941238760948181, 0.02884155884385109, -0.18967415392398834, 1.0822312831878662, -0.033700354397296906, -0.4736277163028717, -1.3037195205688477, -0.49306392669677734, 0.33949369192123413, 0.24377001821994781, -1.2686469554901123, -0.6142304539680481, 1.3938014507293701, 0.40555712580680847, -0.36734268069267273, 0.07654803991317749, 0.9039627313613892, -1.8208595514297485, -0.12961392104625702, 0.38003844022750854, 1.3553229570388794, -0.361970990896225, -0.7144873738288879, 0.9555976986885071, -1.2290273904800415, -0.22281591594219208, -0.0031712797936052084, 1.416826844215393, 2.537301778793335, -1.1373481750488281, 0.3263436555862427, -0.022557789459824562, 0.8329468965530396, 1.0896611213684082, 1.886899471282959, 2.5451245307922363, -0.8288788795471191, 0.5728545784950256, 0.476345032453537, 0.6557974815368652, 0.7817869186401367, -0.2726646065711975, -2.3477227687835693, -1.0607378482818604, -0.10021261870861053, -0.09582577645778656, 0.3018213212490082, -0.7212010025978088, -0.5300793051719666, -1.5510964393615723, 0.5182313919067383, -0.15970292687416077, 0.1743880808353424, -1.3325023651123047, 1.655316710472107, -0.4692411422729492, 1.210584044456482, -0.26259735226631165, 0.1557081639766693, -1.3105559349060059, -0.3712715208530426, 1.0894253253936768, -0.2554197609424591, -2.013773202896118, 0.8598114252090454, -0.6886335611343384, 0.037246037274599075, 1.44637930393219, -0.10447981208562851, -0.5481396317481995, -0.6341685652732849, 0.7301408648490906, -1.6094191074371338, -0.8007358908653259, -0.6065948009490967, -1.237985610961914, -0.8233951926231384, 0.6107190847396851, -0.37426525354385376, 1.1921474933624268, -0.8838724493980408, 1.1363112926483154, 0.9330143332481384, 0.5431236028671265, 0.628057062625885, 1.0164991617202759, 0.09619057178497314, 0.6824700832366943, 2.2762649059295654, -1.245786190032959, 0.8747230172157288, -0.842515230178833, 0.5991949439048767, 0.5637621283531189, -0.2740938663482666, -0.9470276236534119, -0.7814092040061951, 1.2826547622680664, 0.426741361618042, 0.6138455867767334, -0.13298654556274414, -0.13593515753746033, 0.8728196620941162, -0.8193633556365967, 0.22734381258487701, -0.7700172066688538, -1.46324622631073, -0.4392736554145813, 0.42299938201904297, -0.19803734123706818, -0.046794723719358444, 0.11200545728206635, -2.0395030975341797, -1.5495574474334717, 1.0890122652053833, -0.6211979985237122, -0.5800130367279053, -0.29991891980171204, 0.4806858003139496, -1.1632195711135864, 0.5430880188941956, -0.7042397856712341, 0.3620501160621643, 0.7830955386161804, 1.254197597503662, -0.9326159954071045, -0.8070275783538818, 0.449524849653244, -0.20782586932182312, 0.06902662664651871, -0.16473421454429626, -1.0683557987213135, 0.369183212518692, 0.5530372262001038, -0.02767595835030079, -0.6412205100059509, -0.8399443626403809, 1.3926222324371338, 0.3901970684528351, 2.5619254112243652, 2.0272324085235596, -0.5498247146606445, -1.8471906185150146, 0.6691359281539917, -0.22782787680625916, -0.08325299620628357, -0.2797796130180359, -0.6274524927139282, -0.23334935307502747, 1.1272169351577759, -1.5485063791275024, 1.5519105195999146, -0.730683445930481, 0.9061894416809082, -1.3163485527038574, -0.5651996731758118, 1.0065664052963257, -0.10486111789941788, -1.2150545120239258, -0.14782743155956268, -0.6647977828979492, -0.8366715908050537, 1.0200659036636353, 0.050261240452528, 0.6874184012413025, 0.6104611754417419, -0.647189736366272, -1.0800447463989258, 0.8389042615890503, -0.5396112203598022, -0.9854662418365479, 0.11750498414039612, -0.7807637453079224, -0.5957266688346863, -1.9114478826522827, -0.28360894322395325, -1.1557059288024902, 0.683856189250946, 0.29294031858444214, -1.7792898416519165, 0.8653681874275208, -0.8239486217498779, -0.7425252795219421, 0.13789333403110504, 0.07134531438350677, -0.1419423371553421, -1.2408888339996338, -0.34625962376594543, -0.11033742874860764, -0.5447437167167664, -1.5873386859893799, 2.22537899017334, 1.067501187324524, -1.164726734161377, -0.14906759560108185, 0.6822059750556946, -0.5061062574386597, -0.3023170828819275, 0.7627149224281311, 0.4165861904621124, 1.1252447366714478, 0.07319258153438568, -1.4567656517028809, 0.4710063934326172, 1.609682321548462, -1.112019658088684, -0.07253287732601166, 1.0476226806640625, 1.5282104015350342, 0.8603878021240234, 1.2613754272460938, -1.2294150590896606, 2.280323028564453, 0.42887866497039795, 1.2397209405899048, 0.8010143637657166, -0.3254709541797638, -0.5343757271766663, 2.1773812770843506, -1.4405311346054077, 0.6172899603843689, -1.2613130807876587, 0.7460673451423645, -0.8286043405532837, 0.4342281222343445, -0.11252370476722717, 0.948018491268158, -0.04032086953520775, 0.8527339696884155, -1.0903466939926147, -0.2598813772201538, -2.236269950866699, 0.906741738319397, -1.8040761947631836, -0.753696620464325, -0.4520874321460724, -0.7292523384094238, -2.472991466522217, -1.151426076889038, -0.9821922183036804, -0.7683077454566956, -0.61907559633255, -0.8650892972946167, -0.847404956817627, 1.472786545753479, 2.0987439155578613, -1.4241243600845337, -1.1684738397598267, -0.2438049614429474, 0.6894416213035583, -0.523946225643158, -0.6689966917037964, 1.1649117469787598, 0.8519505262374878, -0.9394914507865906, -0.9207018613815308, 1.125739336013794, 0.3056401312351227, 0.12390651553869247, 0.40717923641204834, -1.689376950263977, 0.622898519039154, 1.2820473909378052, -0.831577479839325, -0.07777522504329681, 0.4475647509098053, 1.2665499448776245, -1.4181923866271973, 1.49956214427948, -0.4396693706512451, 1.0603781938552856, -0.31150853633880615, -1.486419677734375, 2.224921226501465, -1.083825707435608, -0.11391746997833252, 1.7456531524658203, -1.2950897216796875, -1.0737043619155884, 0.3998856246471405, 1.7645692825317383, -1.0112797021865845, 1.6930676698684692, -1.039130687713623, -0.45016875863075256, -0.8751027584075928, 0.05518956109881401, -1.8337008953094482, 0.6136805415153503, -0.002582018030807376, -0.21877655386924744, -0.29244136810302734, 0.08981778472661972, -0.6829090118408203, 0.2039843648672104, -0.22174109518527985, 2.443533420562744, -1.3706144094467163, -0.6708161234855652, -0.5067375302314758, -0.4586695432662964, 0.7413449287414551, 0.41988956928253174, 0.6308813691139221, 0.1380317360162735, -0.8761628270149231, 0.18815690279006958, 0.6210185289382935, -1.2393136024475098, 0.899210512638092, -0.12478052079677582, -0.30080798268318176, 3.53275465965271, -2.3029348850250244, 0.6804038882255554, 0.5489122867584229, 0.3339942991733551, -1.0217348337173462, -0.9102562665939331, 1.4452346563339233, 0.25865837931632996, 0.16278298199176788, 0.33668646216392517, 0.5325552821159363, -0.6849088668823242, 1.2420216798782349, 0.2564942240715027, -2.26218581199646, 1.5801423788070679, -0.5149943232536316, -0.7411077618598938, -0.9028072953224182, 0.41580092906951904, 0.737267792224884, -0.5835961103439331, -0.8827463984489441, 0.3847914934158325, -0.29565802216529846, -0.33293208479881287, 0.10048859566450119, 0.2493857890367508, -1.2017605304718018, -1.593018889427185, -0.8988887071609497, 0.6166036128997803, 0.28414013981819153, -0.22204703092575073, -0.6063748598098755, 0.7323618531227112, -0.5728812217712402, 1.0471936464309692, -0.011695628985762596, 1.1187856197357178, 0.3500416874885559, -1.2082363367080688, 0.9761545062065125, -0.6898946166038513, -1.0949853658676147, 1.2691978216171265, 0.28419533371925354, -1.6357396841049194, 1.3521720170974731, -1.6170799732208252, 0.7890327572822571, -0.048834819346666336, -0.054725486785173416, -1.7835091352462769, 0.06150996312499046, -0.7577938437461853, -0.3891787528991699, -1.0483393669128418, 2.2577171325683594, -0.29673174023628235, 0.12821468710899353, -1.7946248054504395, 0.7304271459579468, 0.9746517539024353, 0.21330325305461884, -0.5570197105407715, -1.4007019996643066, 0.4491715431213379, -0.8764932751655579, 0.042113713920116425, 0.9260711073875427, 0.2712162137031555, -0.07034678012132645, 0.31595274806022644, -0.6595350503921509, 1.1247432231903076, 0.07363921403884888, -0.6050733327865601, -1.9508558511734009, 0.47284120321273804, -0.2101154923439026, -0.599856436252594, -1.325628638267517, 0.8019478917121887, -1.3645058870315552, -1.6912695169448853, 0.5885290503501892, 0.6912850141525269, -0.09725821763277054, -0.7510826587677002, -0.2102893590927124, -0.33481061458587646, -0.43346622586250305, 0.4769969582557678, -0.346263587474823, -0.15849097073078156, -1.2350343465805054, -0.004470037762075663, -0.36589646339416504, -0.9840766787528992, -0.5914866924285889, 0.06880570948123932, -0.826124906539917, -0.9627887010574341, 1.6825565099716187, -0.28927600383758545, 1.892730712890625, -0.8818791508674622, 1.1932127475738525, 1.8083436489105225, 1.0352861881256104, -1.21560800075531, -1.5641406774520874, 0.9500762820243835, 0.19060789048671722, 0.6112416982650757, -0.2392236441373825, 1.0718342065811157, 0.810359001159668, -0.6720678806304932, -0.05175381153821945, 1.8461593389511108, 1.8778228759765625, 0.6863681077957153, 1.6333554983139038, -0.8549591302871704, 0.34383711218833923, 0.5324371457099915, 0.2951951324939728, -0.058175262063741684, 2.9108400344848633, 1.297742486000061, 0.5640262365341187, -0.2439282089471817, 0.396878182888031, -0.5067929029464722, -1.791038990020752, 1.1676628589630127, 1.524901270866394, 0.6341925859451294, -0.12386611849069595, 0.4466089606285095, -1.0462374687194824, 0.21580208837985992, 0.35933390259742737, 0.5494306683540344, 0.5222281217575073, 0.7452660799026489, -1.5895590782165527, -0.4062037169933319, -0.9498937129974365, -0.42515087127685547, -0.03870671987533569, -2.442042350769043, 0.9276754856109619, -0.7101109027862549, -1.9093589782714844, -0.23816494643688202, 0.13541871309280396, 0.3019646406173706, -1.0256237983703613, -1.0393866300582886, -0.8662211298942566, 0.25345736742019653, -0.21611617505550385, -0.5072734355926514, -0.700190544128418, 0.7267913818359375, -0.44710633158683777, -1.170567274093628, 0.11299004405736923, 0.4513466954231262, 0.708690881729126, -0.38356921076774597, -0.11612731218338013, 0.06873639672994614, -0.6080693602561951, 0.4461187720298767, 1.1480085849761963, 0.6163386702537537, -0.827234148979187, -1.6966782808303833, -0.8587721586227417, 2.3969521522521973, -0.6238073110580444, 0.7997561097145081, 0.8028943538665771, -0.34191545844078064, -1.4095255136489868, -0.2444947212934494, 0.6266308426856995, -1.2673348188400269, -0.6193667650222778, -0.5455164313316345, -1.0290212631225586, 1.1147748231887817, -0.03846372291445732, -0.308867484331131, -1.179990530014038, -1.2153958082199097, 1.351293921470642, -2.162388563156128, 1.4194337129592896, 0.16668102145195007, 0.3023914396762848, 2.0531792640686035, 0.6509841084480286, -0.31679123640060425, -0.8042736649513245, -1.6934945583343506, -0.6808314323425293, 0.985245406627655, 0.33123892545700073, -0.43546730279922485, -0.7766456007957458, -0.816917896270752, 0.7858784198760986, -1.3133646249771118, 0.3426012694835663, -0.8437294363975525, 0.9587976336479187, -0.34772661328315735, -0.04850263148546219, -0.7005491256713867, -0.7985075116157532, 1.7189795970916748, -0.8966701030731201, 1.493656039237976, -1.009509563446045, 1.9584940671920776, -0.9487018585205078, 0.31025269627571106, 0.6343401074409485, -1.6644093990325928, 0.1607215851545334, -1.490679144859314, 1.574854850769043, 1.6037237644195557, -0.19165430963039398, -0.5598058104515076, 0.34687989950180054, 0.18495875597000122, 0.40181273221969604, 0.8293298482894897, -0.7846066355705261, 1.086893081665039, -0.3877304196357727, -1.0073665380477905, -0.8289502859115601, 0.8913646936416626, -0.24441155791282654, -2.8599588871002197, -1.863616943359375, 0.16393665969371796, 0.7196030020713806, 1.1073435544967651, -0.3964293897151947, -0.8013064861297607, 0.01476801186800003, 0.0016607334837317467, -0.8719281554222107, 0.004580995533615351, -1.1343984603881836, -1.026727557182312, 0.5992648601531982, -2.006702184677124, 0.47825101017951965, -0.3562963604927063, 0.5850445032119751, 0.39320406317710876, -0.6284080147743225, -1.7306410074234009, -0.715909481048584, -0.7626473903656006, -0.8424449563026428, -0.31583911180496216, 2.095088243484497, -1.388530969619751, -0.26790428161621094, -1.418153166770935, 1.3485159873962402, -1.283713698387146, 0.018843794241547585, 0.15742820501327515, 0.7327309846878052, 0.9390645623207092, 1.4391605854034424, 1.6986703872680664, -0.587881863117218, 1.1872344017028809, 1.5405879020690918, -0.4615742862224579, -0.6482018828392029, 0.1505923718214035, -2.495421886444092, -0.024575792253017426, -1.7957983016967773, -0.9214519262313843, 1.244910717010498, -2.056732416152954, -0.3994852602481842, -1.0305789709091187, 0.003141500288620591, -1.8426618576049805, -0.8129599094390869, 0.9997613430023193, -0.10391978919506073, 1.2476420402526855, 1.6659247875213623, 1.7122422456741333, -0.7265850901603699, -0.7605952024459839, -0.47015437483787537, 0.788824200630188, -0.3377881944179535, 0.972822368144989, -0.4894304573535919, -0.7200709581375122, -0.2760715186595917, 1.2923154830932617, 1.699351191520691, -0.49009159207344055, 0.7306155562400818, 0.41220608353614807, -0.0012178880861029029, -0.13919685781002045, -0.8994124531745911, -0.8596013188362122, 0.15766604244709015, -1.1269915103912354, 0.05850154906511307, -0.5459808111190796, -0.7598267197608948, 0.6141597628593445, 1.1875064373016357, 0.6679967641830444, 0.3059401512145996, -0.01670011319220066, -0.39682912826538086, -0.6162936687469482, -0.6340433359146118, 0.842514157295227, 0.7356955409049988, -0.37892448902130127, 1.8720914125442505, 1.0632609128952026, -0.6844326257705688, 1.0283966064453125, -1.0442564487457275, -1.9069535732269287, -0.4564392566680908, -0.457596093416214, -0.6452510356903076, 0.13769395649433136, -0.42810866236686707, 0.4151756465435028, -0.491830438375473, -0.024973103776574135, 1.4503825902938843, -0.5817339420318604, -1.186472773551941, 3.0014986991882324, -2.4949777126312256, 0.4946920871734619, -0.3146747052669525, 0.5272635817527771, 0.4881223738193512, -1.4414113759994507, -0.43027573823928833, -0.8050728440284729, -1.4177900552749634, -0.370129257440567, -0.6340899467468262, 2.4027206897735596, -0.5177069306373596, 0.4231208562850952, -1.2540111541748047, -0.2250412106513977, 0.1631138175725937, 0.8470423817634583, -0.7541377544403076, 0.18965764343738556, 1.095996618270874, 0.8262772560119629, 0.06565102934837341, 2.937314987182617, 0.5122342705726624, 2.1329798698425293, -1.2734241485595703, 0.9614390730857849, 1.0711898803710938, -0.9614138007164001, -0.5094946026802063, 0.625093400478363, -0.29950255155563354, -0.30499669909477234, -0.17825329303741455, -0.6418848037719727, 0.5130066871643066, -0.1184355691075325, 2.2633018493652344, -1.3532634973526, 1.230531096458435, 1.423557162284851, -1.4608021974563599, 1.0799827575683594, -0.09002336859703064, -1.2212109565734863, -1.382613182067871, -0.1771564781665802, -0.1265120953321457, -1.6218671798706055, 0.3299165368080139, 2.073357343673706, -0.45608431100845337, -0.26578283309936523, 0.7682327628135681, 0.6856268048286438, -0.97780442237854, 1.517401099205017, 0.6708401441574097, 0.8767726421356201, 0.24617935717105865, 0.15199042856693268, 0.4852430820465088, -0.8143926858901978, 0.03066924214363098, -0.9419084191322327, -0.8185800909996033, 0.33966535329818726, -0.8109089136123657, -0.9771748781204224, 0.08559642732143402, -0.07589161396026611, 0.8071937561035156, 0.32298123836517334, 0.40852001309394836, 0.06952117383480072, 1.0656657218933105, -0.9225810170173645, 0.09076366573572159, 1.2961753606796265, 0.05986884981393814, 0.5646824240684509, 1.6682697534561157, -0.0518915094435215, -0.18028642237186432, -0.8405531048774719, 0.3475882112979889, 0.5944374799728394, 0.3649866580963135, 0.29567503929138184, 2.027116537094116, -1.208429217338562, -1.5419000387191772, 0.6062502264976501, -0.3482634723186493, 1.0543164014816284, 1.664823293685913, -0.842195451259613, 0.31321170926094055, 0.384041428565979, 0.004661790560930967, -0.28672194480895996, 0.15649764239788055, -0.9850035905838013, 1.6749366521835327, -0.8689634203910828, -0.6467137336730957, -0.08597097545862198, 1.2217686176300049, -0.7240956425666809, -0.02744581177830696, 1.3328604698181152, 0.23747366666793823, -0.3266362249851227, -0.3619709610939026, -0.32962101697921753, -0.2229364812374115, 0.4983847439289093, 0.14414958655834198, -0.3725913166999817, 1.3163797855377197, -0.5576878786087036, 0.7214654684066772, 2.1640961170196533, 1.7175602912902832, -1.38127601146698, -0.639133095741272, 1.310463309288025, 1.006146788597107, -1.1831809282302856, 1.6269967555999756, -0.6401963829994202, -0.09106891602277756, -0.9041864275932312, -0.11252637207508087, 0.9770086407661438, 1.003083348274231, -0.12470315396785736, -0.5142029523849487, 0.8565545082092285, 0.9807175397872925, -0.6489885449409485, -0.3939559757709503, 0.26338616013526917, 0.37193796038627625, -1.1149269342422485, -0.7319740056991577, 0.349397748708725, -0.21930649876594543, -0.16010484099388123, 0.6970816254615784, -0.15907569229602814, 0.52281254529953, 1.4829567670822144, -0.3727537989616394, 0.47997334599494934, -0.2467566579580307, 0.26538747549057007, 2.7955994606018066, -1.3860574960708618, -0.003224543994292617, 1.0859646797180176, -0.2689480185508728, -0.24496421217918396, -1.757320761680603, -0.38984450697898865, -0.08619638532400131, 1.7645751237869263, 0.559960663318634, 2.2808756828308105, -1.4807590246200562, -1.1893113851547241, 0.04620707035064697, 1.348405361175537, 0.862720787525177, -1.0942727327346802, 1.0361964702606201, 0.7375668287277222, 0.5215016007423401, 1.020263910293579, -0.4883992373943329, -1.0265226364135742, -0.34710025787353516, 0.28877514600753784, -0.6197118163108826, -0.6863903403282166, 1.1337213516235352, 0.3820236325263977, 0.49220937490463257, -0.7962281703948975, -0.6843365430831909, 0.6147895455360413, -0.18732838332653046, 0.6996110081672668, -1.1457535028457642, -1.5769490003585815, 0.6048828363418579, 0.6906580924987793, -0.36834588646888733, -0.6771152019500732, 0.7948439717292786, -0.9829962253570557, -1.0310554504394531, -0.09104731678962708, 0.16231095790863037, 0.39194491505622864, -0.24669043719768524, -0.38742053508758545, -0.7957906126976013, -1.07196044921875, -1.0914868116378784, -0.7508726716041565, -0.6571688055992126, -0.6264066100120544, -1.0838977098464966, 0.5285955667495728, -0.33188101649284363, -1.7025729417800903, -0.3233596682548523, -0.65365070104599, -1.2493340969085693, -0.2605123817920685, -0.1571536511182785, -0.05250898003578186, -0.6199253797531128, -1.128420114517212, 0.738285481929779, -0.6724280118942261], [-1.3964107036590576, -0.25948792695999146, 1.1498005390167236, 0.5030900835990906, 1.1702208518981934, -0.5559582114219666, -0.4684264063835144, -0.2619125247001648, -0.673696756362915, 1.1996068954467773, 0.30019327998161316, 1.0172796249389648, -2.4502105712890625, -0.163031205534935, 1.1675292253494263, -0.857895016670227, -0.3712160289287567, 0.1956208050251007, 0.9424367547035217, 1.4956187009811401, 0.5178694128990173, 2.4172215461730957, -0.5363710522651672, -0.9391074180603027, 0.3795470893383026, -0.5925182104110718, 0.5704049468040466, 0.07815410941839218, -1.0279496908187866, -1.152663230895996, -0.2846468687057495, -0.19398678839206696, 0.15767891705036163, 1.2366244792938232, -0.1609356701374054, -0.4778830409049988, -2.7618491649627686, 1.110077142715454, 1.7918782234191895, 0.31631943583488464, 0.6520242691040039, -0.727990448474884, -0.16171278059482574, -0.48673129081726074, 0.7115070223808289, -0.37452971935272217, -2.2324321269989014, 0.1896485984325409, 0.19155265390872955, -0.04527104273438454, 0.17512081563472748, 1.5704278945922852, 2.0962629318237305, 0.2982169985771179, -0.39225906133651733, 0.050865139812231064, 1.296575903892517, 0.26257386803627014, 0.5165704488754272, 0.28733423352241516, -0.6199032664299011, 0.6512601375579834, 1.9561666250228882, -0.8258979320526123, 0.12395904958248138, 0.9159768223762512, 0.5825289487838745, -1.7123439311981201, -0.16027817130088806, 0.3676215410232544, -0.9428557753562927, 0.6653157472610474, -0.25840047001838684, 0.5893245339393616, 0.8256655931472778, 0.8677796125411987, -0.3285242021083832, 1.9434304237365723, 1.1198740005493164, 0.32548049092292786, 1.406587839126587, -0.21190287172794342, -1.6125895977020264, 0.5585744976997375, 0.7333002686500549, 0.27241188287734985, 0.6157301068305969, 0.30440405011177063, -0.9181513786315918, -0.5180392861366272, -2.2487285137176514, -1.19277024269104, -0.09251812845468521, -0.8681322336196899, 1.698351263999939, 1.2533608675003052, -1.349334478378296, 0.0874239131808281, 0.2795892357826233, -0.1873982697725296, -1.4312310218811035, -2.6236140727996826, 0.16457365453243256, -2.221754550933838, -0.08464963734149933, -2.6818623542785645, 1.3271267414093018, -1.1987344026565552, 0.7921170592308044, -0.16080573201179504, 0.9362360835075378, -1.6848301887512207, 0.42671242356300354, 2.3107962608337402, -0.027430692687630653, 0.7324381470680237, 1.4033995866775513, -0.29408106207847595, -0.5587046146392822, 1.4369672536849976, -1.3546745777130127, 1.8166768550872803, -0.9584568738937378, 0.39289841055870056, -0.5525941848754883, 1.078513741493225, 1.412731647491455, 1.7192809581756592, -1.9197680950164795, -1.4276885986328125, -1.0226678848266602, 0.30421358346939087, -0.22895050048828125, 0.2329677790403366, 0.5877938270568848, -0.5289422273635864, 1.0004496574401855, 0.2056642323732376, -1.6499476432800293, 0.012544576078653336, 1.3484560251235962, -2.6455271244049072, -0.7510855793952942, 0.44889357686042786, -0.0279583428055048, 0.07189346849918365, 0.11430475115776062, 0.7592566609382629, 0.5844774842262268, 0.3290085196495056, 0.04624474421143532, -0.16954782605171204, -0.7039551734924316, 1.0318405628204346, -1.081746220588684, -1.082141637802124, -0.4903866946697235, 0.07414934039115906, -0.7790716886520386, 0.08293257653713226, 0.6380370855331421, -0.17158576846122742, -0.37941431999206543, -0.29117125272750854, 0.18011990189552307, 0.46984532475471497, -2.0605905055999756, 0.0203563142567873, -0.18698757886886597, -0.7431880831718445, 0.10708601772785187, 1.3790853023529053, 1.0547020435333252, 1.4365209341049194, 0.3067830204963684, 1.472036361694336, -0.6195265054702759, 0.23248280584812164, -1.5128446817398071, -1.1771888732910156, 0.2983216643333435, -1.3717961311340332, 2.961195945739746, 1.6879990100860596, -0.5152587890625, 0.8100116848945618, 0.3713719844818115, -0.665168285369873, 0.35112789273262024, -0.053523894399404526, 0.2313203066587448, 1.206131100654602, 0.08991480618715286, -0.7989833354949951, -1.2136735916137695, 0.06439249217510223, -3.143080234527588, 0.23949424922466278, 0.07253538072109222, 0.6159864664077759, -1.092575192451477, -0.289899617433548, -0.8068569302558899, -0.1667676568031311, -3.0266811847686768, 0.35995543003082275, 0.15804879367351532, -0.5095023512840271, -1.437546968460083, 1.0381379127502441, -0.9432517290115356, -2.2189712524414062, -1.0488160848617554, -0.6007922887802124, 0.14552989602088928, -1.8571072816848755, -1.3280783891677856, 0.21503183245658875, 1.0832773447036743, 0.4191523492336273, -0.2194174826145172, -0.4456399083137512, -0.12078036367893219, -0.22163201868534088, 0.714384138584137, -0.3889161944389343, -0.533901572227478, 0.17290660738945007, 0.02878662757575512, -0.7714477181434631, -0.5153247117996216, 0.660252034664154, 1.284054160118103, 0.840565025806427, -0.4372555911540985, 0.18945078551769257, -0.9190186858177185, -0.7831825613975525, -1.7845968008041382, -0.5725970268249512, 0.6009878516197205, -0.3578200936317444, -0.7007364630699158, 0.8049245476722717, 0.17892561852931976, 0.6949960589408875, 0.03858834132552147, -1.1600486040115356, 0.24616605043411255, -1.0643510818481445, -0.43969306349754333, 0.2890812158584595, 0.40307849645614624, -0.3429929316043854, -0.32896313071250916, -0.13384217023849487, 1.096504807472229, 0.8199547529220581, 0.07544200122356415, -0.35561519861221313, 1.0106264352798462, -0.3024885356426239, 1.4578109979629517, 0.8050921559333801, -0.8126032948493958, 0.004699830897152424, -0.4888891577720642, 0.6973313689231873, -0.05354967713356018, 0.6891751289367676, 1.6392008066177368, 0.9650384783744812, 1.22848379611969, 1.1045854091644287, 0.9737923741340637, -0.7010874152183533, -0.32490769028663635, 0.10732710361480713, 1.0179952383041382, 0.22451917827129364, 0.15716435015201569, -0.4682004153728485, 2.052253007888794, -0.12913359701633453, -0.6952657103538513, 2.2776830196380615, 0.11290460079908371, 1.146965742111206, -0.2543146014213562, -1.3921865224838257, -1.126228928565979, -1.2472608089447021, -0.5437754392623901, -1.0200468301773071, -1.0197057723999023, 0.8922293186187744, 0.5413166880607605, -0.5494333505630493, -0.5946975946426392, 0.1569906771183014, -0.7324996590614319, 2.101792812347412, -1.180814266204834, -0.6015440225601196, -1.2996304035186768, 0.1716676950454712, 1.6066340208053589, 0.579306423664093, -0.8724113702774048, 0.3404123783111572, -1.4886482954025269, 1.3924462795257568, 0.477318674325943, -0.12827780842781067, 1.5385277271270752, 0.4478304386138916, -0.757723331451416, 0.33737990260124207, 0.015239506028592587, -1.6116257905960083, -0.6247619986534119, 0.7697578072547913, 0.6910563707351685, -0.295176237821579, 0.2858744263648987, -0.32168856263160706, 0.09009730815887451, 0.354076087474823, 0.6590544581413269, 0.26449140906333923, 0.5916253924369812, 0.9279965758323669, -0.7999529838562012, 0.23727059364318848, 0.7190646529197693, 1.1485930681228638, -0.1282537877559662, 0.10447172820568085, -0.9202336668968201, 0.09756974875926971, -0.7393017411231995, 0.6569087505340576, 0.6350237131118774, 1.9272308349609375, -1.9607311487197876, -1.0223499536514282, -1.5371977090835571, -1.5063447952270508, -0.6205787658691406, 0.8328369855880737, -0.7458595633506775, -0.5219866633415222, -1.6292059421539307, 0.14460070431232452, 0.36476847529411316, 0.3448309600353241, 0.7666615843772888, -1.4465079307556152, -0.43998244404792786, 1.0434937477111816, 2.811857223510742, -0.018031517043709755, 1.0537687540054321, -0.881011962890625, 0.47448253631591797, -0.3006401062011719, -0.7851702570915222, 0.8044770956039429, 1.6378626823425293, -1.0964518785476685, -1.1417864561080933, 1.9440163373947144, 1.6671350002288818, 0.00830088835209608, 0.2854152321815491, -1.1166266202926636, 0.47720006108283997, -0.08384495228528976, 0.041261959820985794, 0.23294538259506226, -1.462273359298706, 0.14210179448127747, 2.3310513496398926, 1.3795768022537231, 0.7062864303588867, -1.1553432941436768, -0.17948971688747406, -1.2673544883728027, 0.2834177315235138, -0.6731463670730591, -1.263232707977295, 0.7129836082458496, -2.5202088356018066, -1.7535613775253296, 0.10026171058416367, -0.25166720151901245, -0.5691246390342712, -0.26487985253334045, -0.7606350779533386, -0.08541226387023926, -0.2756146788597107, 0.7099037170410156, -0.4645281732082367, 2.2302212715148926, 0.3857128322124481, -0.6954605579376221, 1.1367055177688599, 1.3557102680206299, 0.3652424216270447, 2.1520423889160156, 0.4096044898033142, -0.07723540812730789, -1.3085781335830688, -0.31868159770965576, -0.179491326212883, -0.36092057824134827, 0.3459504246711731, 0.44121596217155457, -0.3607686758041382, -1.7712492942810059, 0.04897686094045639, -1.6962772607803345, 0.6575533747673035, -0.5717636346817017, 1.5484602451324463, 0.1431005299091339, -0.515485405921936, 0.4572245180606842, -0.13555656373500824, -1.3632280826568604, -0.48667111992836, 1.0270824432373047, 1.1659239530563354, -0.5724865794181824, -0.30183929204940796, 0.47703149914741516, -0.39960891008377075, -1.7847340106964111, -0.11977758258581161, -0.8991966843605042, -2.0186023712158203, 0.39608123898506165, -1.9430540800094604, -0.5610569715499878, 1.3705856800079346, 0.9963412284851074, 1.4584705829620361, 0.27069202065467834, 0.35644182562828064, -0.5541897416114807, -0.1305546760559082, -0.062244150787591934, 0.32624343037605286, 0.42587313055992126, -0.40259483456611633, 1.2219347953796387, 0.08059626817703247, -1.176639437675476, 1.2463890314102173, 1.686334490776062, 0.6177893877029419, 0.343487024307251, -0.3539162278175354, 0.32268625497817993, -1.3694185018539429, -0.38725608587265015, 0.27976566553115845, 1.5270698070526123, -1.1178762912750244, 1.5519558191299438, 0.11050787568092346, 0.4010853171348572, 1.8853261470794678, 1.465166687965393, 0.43572738766670227, 0.5048931837081909, -0.5728006362915039, -2.1296801567077637, 0.2271549254655838, -0.03843875229358673, -0.5572586059570312, -0.5509727001190186, -1.8585542440414429, 0.29786810278892517, 0.3962365388870239, -1.0902704000473022, 0.09548481553792953, 0.47768864035606384, 0.3127230107784271, 0.37996312975883484, -0.56118243932724, 0.18320053815841675, 1.1210983991622925, 0.8894417881965637, 2.0305933952331543, -0.8793419599533081, 0.8804271817207336, 1.0164533853530884, 1.1050772666931152, -1.0858978033065796, 1.1235333681106567, 0.24901461601257324, -0.33318445086479187, -0.6079833507537842, 0.18699321150779724, -0.7097002863883972, -1.2806845903396606, 1.2487198114395142, -1.1501132249832153, -0.6505371332168579, -1.0707409381866455, 0.6840156316757202, 0.7512468099594116, 1.034866452217102, -0.1814241260290146, 1.264504075050354, -0.7455894351005554, -1.098392367362976, 3.0759856700897217, -0.8689732551574707, -0.7084035277366638, -1.5396618843078613, 0.7881913185119629, -0.7100555896759033, 1.5735746622085571, 0.764094889163971, 0.7295325994491577, 0.5450825691223145, -1.7911838293075562, -0.08430829644203186, 0.7165243625640869, -0.5523656010627747, -1.4171611070632935, 0.20910365879535675, -0.8739850521087646, -0.7190181612968445, 0.8697342276573181, -1.807878017425537, -1.641179084777832, -0.6398470997810364, 0.25332391262054443, -0.4947368800640106, 1.2424359321594238, 0.7063192129135132, -0.9747553467750549, 0.224368155002594, 1.2530735731124878, -0.5146982073783875, -0.11599623411893845, 0.19805210828781128, 0.9530604481697083, 1.2284150123596191, -1.4278231859207153, 0.4409223198890686, 1.3228943347930908, 2.355292797088623, -0.7526893019676208, 0.05456006899476051, 0.47541946172714233, 0.18442785739898682, -0.3651001453399658, 1.869301199913025, 0.7882012724876404, -1.6629849672317505, -1.7442314624786377, 0.21114231646060944, -0.1998828798532486, -0.937895655632019, 0.4419157803058624, 0.5934456586837769, -1.4284590482711792, -0.8641800284385681, -1.8580514192581177, -0.10603700578212738, 0.05129867047071457, 1.037575364112854, 1.109688401222229, 1.1158761978149414, -0.1485825777053833, -0.07274235785007477, 1.28778076171875, -0.5383718013763428, 0.2501113712787628, 0.5341327786445618, -0.4425218403339386, 0.813422441482544, 0.26628169417381287, 0.8144547343254089, 1.041856050491333, 0.6141976714134216, 1.1507701873779297, -0.2070079892873764, 0.3159269392490387, 2.206326961517334, -0.9089961051940918, 0.5124742984771729, 1.0371177196502686, -1.8965163230895996, -0.5907313227653503, 0.5493232011795044, 1.321768879890442, -1.2081835269927979, 0.34546616673469543, -0.48323753476142883, 1.3093451261520386, 1.0902787446975708, 0.05081766098737717, 0.6295896768569946, -0.18177133798599243, 1.7159368991851807, -1.1453592777252197, -0.2977442443370819, -0.5689134001731873, 0.9981787800788879, 0.6187103986740112, -0.056593816727399826, 0.7110816836357117, -0.4827333688735962, -0.18560856580734253, 0.10328929871320724, -1.1178600788116455, -0.04725485295057297, -0.22488373517990112, -0.3244558870792389, 0.9598549604415894, 0.003609063569456339, 0.788214385509491, 0.5910195708274841, 0.8440962433815002, -1.443096399307251, 0.033912211656570435, 0.6782441735267639, -1.236838459968567, 0.13445772230625153, -0.44334718585014343, 1.455630898475647, 0.8208767771720886, 0.22680550813674927, -0.006580736022442579, 1.116852045059204, -0.9169512391090393, -0.3681841790676117, 0.8653937578201294, 0.9919682741165161, 0.7850810885429382, 0.798220157623291, 0.9805715680122375, 0.4501722753047943, 0.37303638458251953, -1.1274548768997192, -0.6583558320999146, 0.9596868753433228, 1.0745186805725098, 1.5405391454696655, -1.0058655738830566, 0.38944756984710693, -1.2463505268096924, 1.2392544746398926, 0.9051578044891357, -0.16831304132938385, 0.9381700754165649, -0.5158630013465881, -0.7880247235298157, 1.616868019104004, -0.7275816202163696, -0.06687285751104355, 0.2610672116279602, -0.9332653284072876, -0.6773830652236938, 0.9027122855186462, -1.6876875162124634, 0.46107953786849976, -0.15813711285591125, -0.9899820685386658, -0.16233031451702118, -0.6569255590438843, 0.035674553364515305, 0.20853866636753082, 0.06550265103578568, 0.2559494972229004, 0.30477073788642883, -0.5131005644798279, -0.11567722260951996, 0.7227565050125122, -0.676213264465332, 1.9219872951507568, 0.48124417662620544, -0.8078218698501587, -2.1128220558166504, 1.46857488155365, -0.1823362410068512, -0.38137951493263245, -1.469102144241333, 0.04800734296441078, 0.416919082403183, -1.3767699003219604, -1.1893441677093506, 0.6832176446914673, -0.5701493620872498, 1.372396469116211, 0.2604050934314728, -0.6080037355422974, 1.4801608324050903, -0.7972912192344666, -0.7814978361129761, -0.33942967653274536, 0.38247451186180115, 0.09147930890321732, -2.0718994140625, -1.938124656677246, 0.092561736702919, -0.5410938262939453, 0.7373541593551636, 0.044247355312108994, 1.4466179609298706, 0.12062276154756546, 0.061989329755306244, -1.9797782897949219, 1.5092025995254517, -0.34448981285095215, 0.2181950807571411, -0.40353822708129883, -1.215402364730835, -1.353570580482483, -0.8026608824729919, -0.3140351474285126, -0.9154981374740601, -0.42619773745536804, -1.3167632818222046, 0.11192675679922104, -0.35628360509872437, -1.258732795715332, -0.322524756193161, 0.7792701721191406, 0.4359913170337677, 1.4999229907989502, -1.9329657554626465, -2.008596658706665, 0.4073151648044586, 0.6479810476303101, -0.33914145827293396, -0.9730596542358398, -0.011980175971984863, -1.466009497642517, 0.5273708701133728, -1.4652522802352905, 0.2385031282901764, 0.53692227602005, 0.19914937019348145, 0.7355749011039734, 0.061673905700445175, 1.3288275003433228, -0.4357374906539917, 1.7809172868728638, 0.451901912689209, 1.392393946647644, 0.4982346296310425, -0.4963071048259735, -1.0789986848831177, 0.5229454040527344, 0.4368836581707001, 0.36925873160362244, -1.7340068817138672, -2.0202672481536865, 0.6446854472160339, 0.4158232510089874, -0.3896227180957794, -2.098865032196045, -2.3788905143737793, -1.5576040744781494, -0.929438054561615, -1.284460425376892, -1.3131299018859863, 1.0633214712142944, -0.9466385245323181, -1.4903756380081177, 0.027003666386008263, 1.4887281656265259, 1.1326699256896973, 1.0897026062011719, -2.160088539123535, 2.2136940956115723, 0.41087836027145386, -1.2945116758346558, 1.585015058517456, -0.20731620490550995, 0.743828535079956, 0.235390305519104, 0.5225232243537903, -1.231880784034729, -1.7634108066558838, 0.2939005196094513, 0.34131765365600586, -0.9995660185813904, -0.6870177388191223, -0.24433355033397675, -0.2779698371887207, -0.4223516285419464, 0.4589008390903473, 1.022851824760437, 1.380998969078064, -0.43058422207832336, 0.23262140154838562, -1.7561814785003662, 0.7098116874694824, 0.4121825098991394, 0.4918220043182373, -1.0953055620193481, 0.08489964157342911, -1.4420324563980103, 0.8629754185676575, 0.5753465890884399, 1.0635912418365479, 0.41760969161987305, -0.5729096531867981, 2.3332226276397705, 1.378504753112793, -0.29880809783935547, 1.2419687509536743, -1.479522466659546, -1.29497492313385, -1.1191562414169312, -1.1160693168640137, 0.4722326695919037, 0.4236712157726288, 1.0676203966140747, -0.09034834057092667, -0.372575581073761, 0.024742145091295242, -1.107298731803894, -0.44152358174324036, -0.12301523238420486, 1.1936384439468384, -1.7414034605026245, 1.1010634899139404, 0.17527729272842407, -0.2846408784389496, 0.20301689207553864, -0.3497369885444641, -0.38708293437957764, -0.7827817797660828, -0.8439664244651794, 1.7340577840805054, 0.2487320452928543, -0.24669770896434784, 0.4023406505584717, -1.9531590938568115, 0.08351068943738937, 0.18759486079216003, 1.5026624202728271, -1.6231555938720703, 0.3206454813480377, -1.122859001159668, 1.1681392192840576, -1.0388853549957275, -1.6133368015289307, -0.44752275943756104, 0.6897134780883789, -0.4355272650718689, 1.3451998233795166, -1.1569170951843262, -0.6999425292015076, 0.12024573236703873, 0.5393707752227783, -0.3040626049041748, -1.0759135484695435, -0.022873209789395332, 0.3431689143180847, 1.9319392442703247, -0.8971689939498901, -1.700566053390503, -0.7046502828598022, -1.7676265239715576, 1.4138541221618652, 0.4219430983066559, -0.47004279494285583, -0.5001875162124634, -0.5430166125297546, -0.9641274213790894, -0.7364447712898254, 0.4173540472984314, -0.17594949901103973, -0.0941232442855835, -0.16852140426635742, -1.003804087638855, 0.6106475591659546, -0.34976109862327576, 1.8888394832611084, 1.0021748542785645, 0.7587186694145203, -1.0480812788009644, 0.2862333655357361, -0.0005062103737145662, -0.9510146975517273, 1.5591081380844116, -1.1549712419509888, -0.37669891119003296, -0.791339099407196, 0.8596042990684509, 0.13255834579467773, 0.13634692132472992, -0.8000706434249878, 0.9476909041404724, -0.9623556137084961, 0.5943372249603271, 0.8971331715583801, 0.9906915426254272, 2.2666656970977783, 0.7004618644714355, -0.3794967830181122, 0.24735334515571594, -1.5443453788757324, -1.5602285861968994, 0.4001091420650482, 2.2012805938720703, 0.6129013299942017, 0.9838388562202454, -1.2080270051956177, -0.1227261871099472, -0.6118391156196594, -1.2036187648773193, -0.32593297958374023, 1.855678915977478, 1.325379490852356, 2.7709662914276123, -1.4506115913391113, 1.6799815893173218, -0.0793023630976677, 0.01157489512115717, -0.44430989027023315, 0.1630614697933197, -1.192286729812622, 1.20396089553833, 0.19818295538425446, -0.9997052550315857, -0.8767279982566833, -0.788011372089386, -0.7555800080299377, -0.17801280319690704, 0.30992192029953003, -0.24744923412799835, 1.4818401336669922, -0.5260736346244812, -0.8540765047073364, 0.3188537061214447, 1.2036621570587158, -2.178415060043335, 1.1782320737838745, -1.5782352685928345, 0.5466557145118713, -0.7764295339584351, 0.8491451740264893, -0.7432748675346375, 0.7101057767868042, -0.031747449189424515, -1.1000574827194214, 3.421388864517212, 0.03402990102767944, -0.18441122770309448, 0.022969549521803856, -0.41263091564178467, -0.21268194913864136, -0.936942458152771, 0.2359018623828888, -0.7649999856948853, 0.5651200413703918, -0.497766375541687, -0.5021836161613464, 1.22633957862854, 0.6338598132133484, -0.5787721872329712, 2.0520031452178955, -0.005701357964426279, 1.4245572090148926, -1.4764735698699951, -0.4300336241722107, -0.557284951210022, -0.015944214537739754, -0.3678474724292755, 1.5468816757202148, -0.3109198808670044, -0.6202924251556396, 0.06337905675172806, 0.217307448387146, 0.042131949216127396, -0.14578546583652496, -0.7834315896034241, -0.3568568527698517, 1.441010594367981, 0.18598665297031403, 1.1503609418869019, 0.05185580998659134, 0.607894241809845, 0.9319934248924255, -0.11962951719760895, 0.8313912749290466, -0.6232025623321533, 0.7444381713867188, -0.1609007567167282, 0.8741738200187683, -0.21790488064289093, 0.5699445009231567, 0.8978422284126282, -1.9541120529174805, -1.0730289220809937, 0.06798728555440903, 0.8834612369537354, 0.12936319410800934, -1.8240020275115967, -0.8878110647201538, 0.6989771127700806, 0.03482582047581673, 0.6961305141448975, -1.2955777645111084, -0.182486429810524, -0.5150707960128784, 1.1134015321731567, 0.1212683841586113, -0.9963074922561646, -0.37280285358428955, -0.37980830669403076, -0.07263150811195374, -0.8814269304275513, 0.9452157616615295], [-0.8819548487663269, 0.21760429441928864, 1.6818968057632446, 0.10907640308141708, -1.8163442611694336, 0.11631609499454498, 0.4572964608669281, 0.20363852381706238, -0.6316903233528137, 1.320757508277893, -0.7938330173492432, -0.06293685734272003, 0.33818280696868896, 2.386800765991211, -0.9187626838684082, 0.3211290240287781, 0.9326307773590088, 0.5887445211410522, -1.1369662284851074, 2.408759355545044, 0.8328951001167297, 0.055170658975839615, -0.32518139481544495, 2.456021785736084, 0.262568861246109, -1.9205877780914307, -1.0335456132888794, 0.608199417591095, -0.5900981426239014, 0.44762247800827026, 0.5252515077590942, -0.046796832233667374, -0.4939686954021454, 0.759618878364563, -0.7622466087341309, 0.6388528943061829, 0.7658599019050598, -0.5179973244667053, 1.6713255643844604, 1.022450566291809, -1.9270786046981812, 1.0282390117645264, -0.6764188408851624, -1.0315366983413696, -1.0151474475860596, -1.2149204015731812, -0.33710506558418274, -1.9952417612075806, 0.7591492533683777, -1.323197364807129, 0.05990856885910034, -0.9176394939422607, -1.737307071685791, 0.7612445950508118, -0.08163278549909592, -0.38963449001312256, -1.7710909843444824, 1.7313423156738281, 1.0473308563232422, 0.6584871411323547, -0.5355359315872192, 2.1628456115722656, 0.7307303547859192, -1.3085204362869263, 0.9877601861953735, -0.7126138210296631, 2.0370333194732666, 1.4629684686660767, 0.7602307200431824, -0.535020649433136, -0.4056239724159241, 0.6419291496276855, -0.32755252718925476, 0.9992774128913879, 0.44685840606689453, -0.01698465272784233, 0.9892953634262085, -0.49191391468048096, -1.1211137771606445, -0.42967063188552856, 0.6512205004692078, 2.474043846130371, -1.738834023475647, -0.629455029964447, 0.8783365488052368, -1.51351797580719, 0.33963197469711304, -1.0610092878341675, 0.014129209332168102, -0.43961387872695923, 1.0299206972122192, -1.2784024477005005, 1.5417805910110474, 0.3028494715690613, -1.6277238130569458, 0.18719792366027832, 0.30085843801498413, 1.0991263389587402, -0.6005534529685974, -1.7453500032424927, 0.40745601058006287, 0.7145798206329346, 0.5754406452178955, -0.16901303827762604, 1.189440131187439, -0.12449941039085388, 0.2513556182384491, 0.38520628213882446, 1.7568645477294922, -0.4013305902481079, 0.13780316710472107, 2.4386942386627197, 1.1680068969726562, -0.199479341506958, -0.9686644673347473, -1.0219734907150269, 0.7954806089401245, 0.32415536046028137, 0.6494214534759521, 0.8033536672592163, -0.08800943195819855, 0.18883772194385529, -0.3157443702220917, 0.6118165254592896, -1.5896111726760864, -1.0973753929138184, -1.437227487564087, -0.667362630367279, 1.34874427318573, -0.2398095577955246, -2.7678608894348145, 0.9031549096107483, 0.26242390275001526, -1.0582722425460815, 0.47889244556427, 0.9486424922943115, -1.4008105993270874, -1.783850073814392, -0.9521551132202148, -0.2539482116699219, -0.024144405499100685, -1.4711600542068481, 1.2697324752807617, -1.488160490989685, -0.8403849601745605, -1.346235752105713, -0.019497785717248917, -2.0113234519958496, 0.6472591161727905, -0.0893251895904541, 2.042841672897339, 0.45750918984413147, -0.04760114103555679, -3.219964027404785, -0.7974249720573425, -1.1217098236083984, -1.6035704612731934, 0.6990271806716919, -1.0910394191741943, -1.4764056205749512, -0.049328260123729706, 0.5531003475189209, -1.5441299676895142, -1.1048458814620972, -0.1408771574497223, -1.8806779384613037, 1.1269313097000122, 0.21809369325637817, 0.10286590456962585, 0.1105942502617836, 0.11864737421274185, 0.00734518188983202, 1.7153589725494385, -0.35298848152160645, -1.2277052402496338, -0.24326537549495697, 0.20852848887443542, -1.7708096504211426, -0.007003448903560638, 0.9728326797485352, 1.223463535308838, 0.4702383875846863, 1.069253921508789, -0.7616833448410034, -2.475710153579712, 0.6494300961494446, -1.0759291648864746, -1.1025277376174927, 1.0757681131362915, -0.3726757764816284, -1.108939528465271, -0.5946652889251709, 1.0050163269042969, -0.6856350898742676, 0.3228631913661957, 0.4811238944530487, 2.37050199508667, -1.1164473295211792, 1.4972240924835205, 0.18949007987976074, 1.5133506059646606, 0.44986027479171753, 0.447713166475296, -0.20612172782421112, 0.21374662220478058, -1.3005450963974, -0.4256255626678467, 1.7452915906906128, 0.8244836330413818, -1.010955572128296, 1.2568289041519165, -1.206527590751648, -0.16997572779655457, -0.5357381105422974, -0.6941041350364685, 0.3693787753582001, 0.8295166492462158, -0.1788553148508072, 2.1434149742126465, 0.04341242089867592, -0.3244827389717102, 1.2922676801681519, -0.5616260170936584, 1.9998283386230469, -1.7390649318695068, -0.0793658122420311, -1.0940303802490234, 1.7380108833312988, 0.14826884865760803, -1.1622902154922485, 0.6946223974227905, 0.49611884355545044, -1.3793083429336548, 0.30305591225624084, -1.2953529357910156, 0.2071738839149475, -1.3694229125976562, 0.016678104177117348, 1.3926489353179932, -0.9324032068252563, 1.2317655086517334, -0.4016483426094055, -0.4989143908023834, -1.6229934692382812, -0.8233604431152344, 1.443047046661377, -0.21964064240455627, -0.7814927697181702, 1.6030343770980835, 0.12314746528863907, -2.087874174118042, 0.28822070360183716, -0.1010197252035141, 0.6344055533409119, 0.5660075545310974, -1.4357081651687622, 0.4508742392063141, -0.11398103833198547, 1.2732875347137451, 1.1334640979766846, 2.1006147861480713, 1.3546924591064453, -0.8280525803565979, 2.465723991394043, 0.923003077507019, -1.9802219867706299, 0.3321908712387085, -0.10479506850242615, 1.6462115049362183, 0.8386082053184509, 0.4757194221019745, 0.7976038455963135, -1.2368186712265015, 1.616883635520935, -0.47244274616241455, 0.16126514971256256, 0.31359484791755676, 0.9630570411682129, 0.9745612740516663, -0.07160773873329163, 0.06956601142883301, 0.16580091416835785, -0.3124115765094757, -1.2236441373825073, 0.09869379550218582, 1.793477177619934, -0.5011277794837952, 0.8015468716621399, -2.225614070892334, -0.9734136462211609, 0.14659668505191803, -1.5255080461502075, 0.4254108965396881, 0.6085026860237122, 0.2518021762371063, 0.5838334560394287, -0.0825423151254654, -0.8114973902702332, -1.461490273475647, 0.20032940804958344, 0.33313000202178955, -0.405923068523407, 1.469340443611145, 0.9097973704338074, -0.6342816352844238, 0.3332507014274597, -0.04257342964410782, -1.176879644393921, 1.6583707332611084, -1.0428194999694824, -0.18141427636146545, 0.18019014596939087, -2.2476587295532227, 1.561452031135559, 1.0536913871765137, 1.4853589534759521, 0.12601694464683533, -0.09424500912427902, -0.20855717360973358, -0.42899855971336365, 0.8483400344848633, 0.7506880164146423, -0.34289389848709106, -0.7541165351867676, 0.0025257919915020466, 1.033195972442627, -0.10404849052429199, 0.3645242154598236, 0.5055453777313232, 2.8176538944244385, -0.8824679255485535, -0.27246353030204773, 0.8056886792182922, -1.3282755613327026, -0.10678335279226303, 1.2559843063354492, 0.13969673216342926, 1.8806159496307373, 0.7178624272346497, 0.759114682674408, -0.9335005879402161, -0.1724797636270523, 0.6765835881233215, -1.062294602394104, 0.03015153855085373, 0.3492352366447449, -0.35396191477775574, -2.439953565597534, 1.0035275220870972, -0.23592431843280792, -1.6268417835235596, 0.6128590106964111, 0.5676574110984802, -0.6634313464164734, -0.4656258225440979, -0.9645894765853882, -0.056024424731731415, -1.8262516260147095, 0.4570728838443756, -0.3904663622379303, 0.178142249584198, 0.8396690487861633, 0.8033969402313232, 2.2489755153656006, -1.4085880517959595, -0.6726880073547363, 1.021299958229065, -0.13001605868339539, -0.7630075812339783, -1.3936368227005005, 0.05872580409049988, 2.6887543201446533, 0.6139823794364929, -0.9456390738487244, -0.0016482928767800331, -0.7290552854537964, -0.5152091979980469, -0.6790653467178345, -1.7450617551803589, 0.6392408013343811, 1.1250886917114258, -0.47281011939048767, -2.202298164367676, -1.2325410842895508, 0.5037269592285156, -0.4504445195198059, -0.5017000436782837, -0.8813651204109192, 1.57283353805542, -0.9184000492095947, -0.829207181930542, -0.09316468238830566, 0.11549150943756104, 0.8923065066337585, 1.000035047531128, 0.06754568964242935, 0.8190734386444092, -0.3673582971096039, -0.48861488699913025, -1.1167711019515991, -2.6771585941314697, 1.2207387685775757, 1.792985439300537, 0.5059909224510193, -1.5397144556045532, 0.6748316884040833, -0.3099743723869324, -1.1349722146987915, 0.884029746055603, -0.45513319969177246, -0.5497400164604187, -2.245617389678955, -1.977768898010254, 2.0298850536346436, -0.8446119427680969, 1.385675072669983, -0.6321393251419067, 1.7450306415557861, 0.893674910068512, 0.5209810733795166, 1.2910829782485962, 0.577670693397522, 1.6214114427566528, -1.5303109884262085, 0.515763521194458, -2.060518741607666, -0.5564696192741394, -2.932352066040039, -0.28258052468299866, -2.4350695610046387, -0.2593368887901306, 0.38666194677352905, -1.6769402027130127, -0.2726324796676636, 1.4444841146469116, 0.026097677648067474, 2.088174819946289, 1.2528895139694214, 1.3108371496200562, -0.27537041902542114, -0.8861745595932007, 0.46974697709083557, 1.07042396068573, 0.18454961478710175, 0.6362603902816772, -0.5008153319358826, 0.7288541793823242, 0.33361679315567017, -0.29853177070617676, -0.8389276266098022, -1.295457363128662, 0.2642729878425598, 0.9282345771789551, 1.484131097793579, 0.07833585143089294, -0.39872199296951294, -0.3802184760570526, -0.03672655299305916, -0.3028016984462738, -1.830592155456543, 0.4070565402507782, -1.5112223625183105, 0.7122579216957092, -0.8175807595252991, -1.0488014221191406, 2.284519672393799, 1.1691325902938843, 0.010123306885361671, -0.366502970457077, 0.043791353702545166, 0.4098745882511139, 1.6930369138717651, 0.7449603080749512, -0.13347892463207245, -1.4049326181411743, -0.34401825070381165, 0.8456195592880249, 0.8290875554084778, 0.7978204488754272, 0.30409321188926697, 0.43430423736572266, -0.3465716242790222, 1.9357600212097168, -0.5913289189338684, -0.43486925959587097, -0.7145310640335083, 2.8582491874694824, -0.39050889015197754, 0.976492702960968, 1.7908796072006226, 1.111211895942688, 0.17800185084342957, -0.5647987127304077, -0.06732267886400223, 0.036181483417749405, -0.0539671927690506, -0.688811719417572, 1.849891185760498, 0.033465560525655746, -0.007012525573372841, 1.7728626728057861, 0.056587252765893936, -0.25096604228019714, 1.3332825899124146, 0.2631314694881439, -1.2508610486984253, -1.0267138481140137, 0.6636001467704773, -0.7070860266685486, 0.2457980513572693, 0.39678478240966797, -0.30326300859451294, -0.1715565323829651, 0.8010485768318176, 0.7326037883758545, 0.07916028797626495, 0.11746394634246826, 0.8376671671867371, 2.6393909454345703, -1.2248095273971558, -1.080331563949585, -0.5439356565475464, -1.8937273025512695, 0.3965126574039459, -0.27358853816986084, 0.833385705947876, 0.45555171370506287, 1.9018889665603638, -0.6870989203453064, -0.9820452928543091, 0.322540819644928, -1.1965581178665161, -0.04172956570982933, 0.572331964969635, 0.20918872952461243, 1.2933086156845093, 0.7016086578369141, -0.029116399586200714, 0.8929793238639832, 0.6122069954872131, -0.3625966012477875, -0.5781112313270569, 0.17058314383029938, -1.0420689582824707, 1.0561248064041138, -0.12833890318870544, -0.49225160479545593, -0.0962430015206337, 1.0482887029647827, -1.1243131160736084, -1.2668834924697876, -0.4645090401172638, -1.150099754333496, 0.2574315667152405, -0.07019540667533875, -1.201953649520874, -0.6296215653419495, -0.2507186532020569, 0.806900680065155, 0.7009605169296265, -1.3649768829345703, -0.39466094970703125, -0.4126068949699402, 1.2248300313949585, -0.26798075437545776, 0.9629209041595459, -0.3504822254180908, -0.0553552471101284, 1.1849441528320312, -0.5027495622634888, 1.2368940114974976, 0.9179078936576843, -0.4114232063293457, 0.7938584685325623, -0.07370023429393768, 0.858801543712616, 1.0023205280303955, -1.7499053478240967, 0.6240019202232361, 0.13606947660446167, -0.7079129219055176, 0.7465538382530212, -0.37237226963043213, 1.2743152379989624, -1.5930519104003906, -1.4155631065368652, 1.589086651802063, 1.1784127950668335, 0.6110119819641113, -0.964836597442627, 0.7837432622909546, -1.438844084739685, -0.6402437686920166, 1.1848537921905518, 0.7596735954284668, -0.12407930195331573, -0.6372941732406616, -0.6347818374633789, -0.21314947307109833, -0.12049421668052673, -1.1904784440994263, -1.6177120208740234, 1.281352162361145, 0.4460965692996979, 0.6424985527992249, 0.14716385304927826, -0.704717755317688, 0.5351508259773254, -1.056275486946106, 0.8347218632698059, -0.04479903355240822, -0.12260591983795166, -0.7356168627738953, -0.5725598335266113, -1.1989609003067017, 0.8187346458435059, -0.768595278263092, 0.8190104961395264, -1.8869318962097168, 0.08164969831705093, 0.017070528119802475, 0.8860867023468018, 0.6048758029937744, -0.14481984078884125, -2.0433552265167236, 1.1565124988555908, 1.2175086736679077, 1.4008246660232544, 1.133949875831604, -0.44838833808898926, -1.097827434539795, -2.072227954864502, -1.2984635829925537, -1.2786716222763062, -0.045771799981594086, 0.22550068795681, -0.9021891355514526, 0.6918981671333313, -0.06709718704223633, 1.2665785551071167, -1.0227100849151611, 0.22953282296657562, 0.2606521546840668, 1.433180570602417, -1.1845382452011108, -0.3316078186035156, -0.46283334493637085, 0.9561794996261597, 0.8852280378341675, -0.10951486229896545, 0.20083650946617126, 0.4521096646785736, -0.5647416710853577, 0.2531731128692627, 0.10173027962446213, -1.1013833284378052, 0.2826705276966095, -1.9098076820373535, -0.8900449275970459, -0.10813591629266739, -1.2901772260665894, -1.0798956155776978, -0.295553982257843, 0.1869262307882309, 0.027967343106865883, -0.44713330268859863, -0.7629978060722351, -1.3339431285858154, 0.11611568927764893, 1.6632370948791504, 0.7219499945640564, 0.6607713103294373, 0.8797112703323364, 0.9474000334739685, 0.2423168569803238, 0.18128904700279236, -0.626650333404541, -0.013971975073218346, 0.2619774341583252, -0.4562438428401947, 1.0171703100204468, 0.4761161208152771, -0.8287822008132935, 0.37203356623649597, -1.2336610555648804, -0.15222011506557465, -0.982443630695343, 1.7686223983764648, 1.4014612436294556, 0.4238727390766144, 1.7308082580566406, 2.306960344314575, -1.128139615058899, -0.391251802444458, 0.902690589427948, 0.4988994896411896, -0.6284465789794922, -0.4862036406993866, 0.4913502037525177, -0.8270184397697449, 0.9727546572685242, 0.9179022312164307, -0.42452943325042725, 2.1215405464172363, -1.3497040271759033, 1.518563151359558, 1.0851402282714844, -0.7327767610549927, -0.6699103713035583, 0.562821626663208, -0.6013972163200378, -0.016445612534880638, 0.8222182393074036, 0.10757394134998322, 1.2012628316879272, 0.7691335082054138, 0.40833255648612976, 0.8434528708457947, 0.2224518209695816, -0.7522672414779663, -0.49350082874298096, 0.3440667688846588, 2.9172184467315674, -0.3018294870853424, 1.854889988899231, -1.2030047178268433, -1.975911259651184, -0.3937487304210663, -1.6110197305679321, 1.3251571655273438, -1.5611355304718018, 0.06705969572067261, -1.0197668075561523, -0.8769272565841675, -0.1431618630886078, 1.6061246395111084, -1.2847793102264404, -1.2106750011444092, 1.3378618955612183, 0.3147391080856323, 1.0121631622314453, -0.3470979928970337, -0.41203323006629944, -0.9646956324577332, 1.5117615461349487, 0.007892691530287266, 2.622239351272583, -0.7172513604164124, -0.5727858543395996, -0.397942453622818, 0.19359777867794037, -0.33643412590026855, 1.5540293455123901, 2.5298385620117188, 0.7099275588989258, 0.04826357215642929, 0.536492645740509, 1.9565129280090332, -0.6280986070632935, 0.5130646228790283, -0.03562341257929802, -0.17421533167362213, -1.1449213027954102, -1.2788960933685303, -0.30353644490242004, 0.5016334652900696, -1.0017204284667969, -0.22440321743488312, 0.08966818451881409, -1.5317994356155396, 0.13603270053863525, 0.6142597794532776, 0.47451406717300415, 0.6777936816215515, -1.0582940578460693, -0.4037706255912781, -2.443288803100586, -1.0551069974899292, -0.9355283975601196, -0.47107797861099243, 0.6726620197296143, 0.1584191471338272, 0.28267863392829895, -1.6815145015716553, -0.31257954239845276, 0.5584976077079773, 0.9907633066177368, 0.38832008838653564, -1.0257041454315186, 0.22870098054409027, 0.47767624258995056, -0.07399044930934906, -0.904228150844574, -1.7613859176635742, 0.22559307515621185, 0.20847010612487793, 0.6075355410575867, 0.5774831771850586, 1.0433136224746704, -0.02535194717347622, 1.0629305839538574, -1.0013904571533203, -1.2079006433486938, -0.5291222333908081, 0.747665286064148, -0.14606043696403503, -0.11915306746959686, -0.1604716181755066, -0.6044427752494812, 0.6565261483192444, 0.5369515419006348, 1.581655502319336, 0.04084928706288338, -0.41635069251060486, -0.846138596534729, 0.08901426196098328, -0.9684915542602539, -0.8519814014434814, -1.7159892320632935, 1.7346267700195312, -1.3059923648834229, -1.520451307296753, -1.7767817974090576, 0.21268436312675476, -0.1700921356678009, -0.1011924222111702, 0.8025728464126587, -0.9734031558036804, 0.3137575685977936, -0.05742260813713074, -0.16920338571071625, -1.4835221767425537, 0.2188704013824463, -0.3383648693561554, -0.29973581433296204, -1.295134425163269, 1.2718654870986938, -0.16349494457244873, -1.1097551584243774, -0.30559128522872925, 0.6583107113838196, 0.07456175237894058, -1.1327555179595947, -1.8628662824630737, 0.17791157960891724, -1.3581207990646362, 0.05152050778269768, -2.82252836227417, 1.8340034484863281, -0.17587785422801971, 0.2676956057548523, -0.6539722681045532, -0.4469810426235199, -1.235095500946045, 0.3286048471927643, 1.48439621925354, -0.1801460087299347, 0.9629373550415039, -0.2665192484855652, -0.25542619824409485, 0.5040147304534912, -2.757822036743164, 0.9398667216300964, 0.770000696182251, 0.40643537044525146, 1.2120338678359985, 0.5501194000244141, 1.26219642162323, 0.3237076699733734, -1.2971409559249878, -1.090954065322876, 0.761921226978302, 2.1952590942382812, -0.5839959979057312, 0.65007084608078, -0.9168750643730164, 0.44896337389945984, 0.233500674366951, -0.2036089152097702, 0.5506406426429749, 0.38173553347587585, 0.6141830682754517, 0.6649655699729919, -0.3554564416408539, -0.44215187430381775, -0.29444241523742676, -1.3787271976470947, 0.3830975294113159, 0.2042282223701477, -0.0451221764087677, -0.8870701193809509, 0.19105033576488495, -0.37827160954475403, -0.8381381630897522, 0.527549684047699, -1.2022465467453003, -1.0993733406066895, 0.7865303158760071, 0.70169597864151, 0.828345775604248, 0.7577317953109741, -0.014706901274621487, 0.6161706447601318, -1.2327511310577393, -0.060035090893507004, -0.07510742545127869, 0.1441412717103958, -0.37349531054496765, 0.7447527050971985, -2.4348466396331787, -0.5110759139060974, 0.28463509678840637, -1.7498502731323242, 1.8175766468048096, 0.5793436169624329, -1.5230146646499634, 0.6068224310874939, 0.15836894512176514, -0.14311274886131287, 1.2373100519180298, 1.0720380544662476, 1.5172191858291626, -2.4386725425720215, 1.8781052827835083, -0.1656172126531601, -0.2858569025993347, -0.5021461844444275, 1.4147065877914429, 1.332413911819458, 0.6568382978439331, 0.34856030344963074, 3.59377384185791, 1.1301432847976685, -1.2889655828475952, -1.899680733680725, -0.5957082509994507, 0.20656028389930725, -0.5262547135353088, -2.326695442199707, 0.5158501863479614, -2.1216118335723877, 0.03569325804710388, -0.9175130128860474, -0.1127684935927391, -0.7004500031471252, -0.5381271243095398, -2.7505462169647217, 0.9666703343391418, 0.39494168758392334, -1.9646055698394775, -1.1433539390563965, 0.3905297815799713, 0.27933719754219055, 0.41782379150390625, 0.24411243200302124, 0.10494767874479294, 0.8313713669776917, 0.5143592953681946, -0.32186421751976013, -0.13377895951271057, -0.27566906809806824, -0.9571433067321777, -1.5060220956802368, -2.082740068435669, 0.17902182042598724, -0.2517924904823303, 0.7066206932067871, -0.9332956075668335, 0.8974742889404297, 1.8736339807510376, -0.11236900091171265, 0.050397276878356934, -0.8124094605445862, 0.373523473739624, 0.9166615605354309, -0.7316442728042603, -0.01422789879143238, 0.35068902373313904, -0.7295238971710205, -2.3868424892425537, -1.392174243927002, -1.5463030338287354, 0.6164326071739197, 0.865771472454071, 1.3008097410202026, -0.1072603240609169, -2.5022058486938477, 0.3396158218383789, -0.25350597500801086, -0.1425837129354477, -0.4724159836769104, -0.3701731264591217, 0.4083964228630066, 1.721739411354065, 1.1727550029754639, 0.9795560836791992, -0.36362379789352417, -1.39173424243927, 0.02849785052239895, -0.18543456494808197, 0.4365885853767395, 1.6739557981491089, -0.9771837592124939, -0.7630098462104797, 1.4177128076553345, -0.9893266558647156, -0.13242365419864655, -1.3500646352767944, 0.0786583200097084, -0.005797409452497959, -1.9284950494766235, -0.5416144132614136, 0.7355270981788635, -0.2748761475086212, -0.5695909261703491, -0.7994059324264526, 1.3657468557357788, 1.6442962884902954, 0.9995476007461548, 0.9458160996437073, 0.8478483557701111, 0.2728043794631958, -0.6303064823150635, -0.33566296100616455], [-0.465328574180603, 0.024800630286335945, 2.255620002746582, -0.26809078454971313, 0.2612057626247406, 0.386389821767807, 0.9297494888305664, -1.4976714849472046, 1.571979284286499, -1.178723692893982, 0.6772662401199341, -0.023221807554364204, 0.6266621947288513, 1.2794033288955688, -0.21301990747451782, -0.4458214342594147, -1.0131810903549194, -0.5211603045463562, 0.16431531310081482, -0.5019744634628296, -0.7813941240310669, -0.5879498720169067, -0.7686492204666138, 0.16908115148544312, 0.43617919087409973, -0.38511961698532104, -1.9041452407836914, -0.2745891213417053, -0.055816400796175, -0.04029190167784691, -1.13473379611969, -0.19688698649406433, 0.9230499267578125, -0.6269225478172302, 0.27142533659935, -1.2701416015625, 1.435652732849121, 0.15187165141105652, 0.18051111698150635, -1.1957980394363403, 0.7523742318153381, 1.9434419870376587, 2.2960243225097656, -0.8268847465515137, 2.008671283721924, 1.6116008758544922, -1.0990885496139526, -2.935741424560547, -0.5972902178764343, -2.4421474933624268, -0.8048581480979919, -0.42446666955947876, -2.3459978103637695, -1.2136560678482056, 0.8959499597549438, -1.4037796258926392, 0.6688380241394043, -0.7849480509757996, -0.6377377510070801, -0.024256333708763123, 0.9795953631401062, 0.7732652425765991, 0.7342330813407898, -0.0020050890743732452, 0.5237574577331543, -1.7124770879745483, -1.1217217445373535, 0.9357426166534424, -0.2855798602104187, 0.9102739095687866, -0.2177601009607315, -1.549145221710205, 1.0662106275558472, 1.133673906326294, 1.4391332864761353, -1.0342856645584106, 1.797407865524292, 1.1857990026474, 1.2867393493652344, 1.8479459285736084, -0.6925378441810608, 0.11172027885913849, -0.6388713717460632, -1.2075002193450928, 0.21243201196193695, 0.04620269685983658, 0.07459568977355957, -0.589112401008606, -0.0544930174946785, -0.011410623788833618, -1.9258887767791748, 1.5267266035079956, -0.7311119437217712, 0.006313359830528498, 0.6351320743560791, 1.143904685974121, 1.0760446786880493, 0.8962563276290894, -2.1760942935943604, 0.11872537434101105, 1.0078587532043457, 0.6169565916061401, -0.8730941414833069, -1.0156975984573364, 1.3118640184402466, -0.34392663836479187, -0.9108461737632751, -0.3672032952308655, 0.21257933974266052, 0.006690372712910175, 1.7927510738372803, -0.6789106130599976, 0.07123574614524841, 0.9735983610153198, -1.423518180847168, -0.2721323072910309, 0.7391117811203003, -1.7520793676376343, 1.114371418952942, -1.3346165418624878, 1.1926199197769165, 1.1807129383087158, 0.571846067905426, 0.2874179780483246, 1.1420273780822754, 0.7088847756385803, 0.28126081824302673, 0.45731720328330994, 0.2075931578874588, -0.8347035050392151, 0.5647264122962952, -1.5860011577606201, -0.39414387941360474, -0.9161275029182434, -0.09491237252950668, 2.6336424350738525, 0.42746564745903015, -1.2977279424667358, -1.5600640773773193, -0.12046293169260025, 0.08602543920278549, -2.092799425125122, -0.7068950533866882, 0.8043791651725769, -0.5075506567955017, 0.08539290726184845, 0.2495228499174118, -0.3089727759361267, -0.8478190898895264, -1.4015687704086304, -0.011610888876020908, -1.2392627000808716, -1.6282529830932617, -1.5223138332366943, 0.8988160490989685, -1.0162791013717651, -0.28633490204811096, 0.1775103658437729, -0.9418772459030151, 0.48739346861839294, -1.8015846014022827, -0.6312337517738342, -0.9004363417625427, 0.27223289012908936, -0.058891188353300095, 2.014096260070801, -0.7934637069702148, -0.8448749780654907, 0.808198869228363, 0.8487548232078552, 0.535110592842102, 1.0890940427780151, 1.8566198348999023, -1.9584048986434937, -0.7717090249061584, -0.42140358686447144, -0.05661282688379288, -0.6201562285423279, -0.4335595369338989, -1.1149948835372925, 0.4188864529132843, 1.7769719362258911, -0.34556150436401367, -1.5309813022613525, -0.21003670990467072, -1.0918956995010376, 0.3434661030769348, 0.4637230634689331, 0.36321961879730225, -1.3160970211029053, -1.3022570610046387, -1.7637746334075928, -2.143342971801758, 0.9934231638908386, -0.0714753046631813, -0.30575376749038696, 1.2011985778808594, 0.09560968726873398, -1.2137939929962158, 2.5117201805114746, 0.8691818118095398, 0.4638843834400177, 0.09536456316709518, 0.5635744333267212, 0.10257688164710999, -0.4305064380168915, -0.8230229616165161, 0.8134617209434509, -0.715417206287384, 2.1401824951171875, -1.2808562517166138, 0.17699317634105682, -0.6246092319488525, -0.28295180201530457, 0.20416925847530365, -0.4331582188606262, -1.5391281843185425, -0.7295660376548767, -0.5926523208618164, -0.3440156579017639, -0.16859488189220428, -0.21839287877082825, -0.12931469082832336, 1.1277538537979126, 0.5214568972587585, 0.1405845433473587, -0.10482148826122284, -0.7060850262641907, 0.6634284853935242, -0.3764512240886688, 1.7705345153808594, -0.010248475708067417, -1.9792169332504272, 0.6936447024345398, -0.6609651446342468, -0.33876916766166687, -0.2851218581199646, -0.16929273307323456, -0.6108490824699402, -0.5111192464828491, -1.0570449829101562, -0.7707272171974182, 0.4681251645088196, 0.14307300746440887, 0.4148317277431488, 0.660839855670929, 0.5829035639762878, -0.3532627820968628, 0.6813787817955017, -2.211883544921875, 0.9131790399551392, 0.2314728945493698, -0.09913367033004761, -0.9206658601760864, 0.3106911778450012, 1.4043112993240356, -1.1143032312393188, -0.28405123949050903, -0.5214072465896606, 0.5583441853523254, -0.34627553820610046, 0.7123553156852722, -0.3647328019142151, 0.6503317356109619, 0.8410167098045349, 0.4709892272949219, -1.1425466537475586, 0.38111722469329834, -0.9618189930915833, -0.31400150060653687, 0.38897955417633057, 0.8542807698249817, -0.32565152645111084, 0.26142048835754395, 0.47598502039909363, -0.15592928230762482, 0.8196682333946228, -0.6700570583343506, -0.27108287811279297, 0.3595143258571625, -1.211176872253418, -0.008129945024847984, 0.8157742023468018, 2.2421975135803223, -0.49042990803718567, -0.1273196041584015, 0.8071805238723755, 0.2556769549846649, -0.3243371546268463, 0.27923548221588135, 1.3004119396209717, -0.15634821355342865, 1.1834865808486938, 0.007229567505419254, 0.3612591624259949, -0.4055590331554413, 0.0457870215177536, -1.015459418296814, -0.932217538356781, 0.2990959882736206, 1.7355177402496338, 1.1536928415298462, 0.34152495861053467, 0.09583669900894165, -0.14449898898601532, -1.3390225172042847, -0.2541055977344513, 1.8012044429779053, 0.7770957946777344, -0.07592374086380005, 0.08942433446645737, 0.5171758532524109, -0.1372237503528595, -0.5348678231239319, -2.272996425628662, 0.8148268461227417, 0.3124457895755768, 0.5620007514953613, -1.216017723083496, -1.6573246717453003, 0.1830345094203949, -0.15252918004989624, 0.5299468636512756, 0.9138559103012085, 0.4112328290939331, -1.272074580192566, 1.1020245552062988, -1.0454562902450562, -0.9700556397438049, -1.1472694873809814, -1.4248696565628052, 0.22429874539375305, 0.4869724214076996, 1.2248338460922241, 0.19549383223056793, 0.8769757747650146, 2.391664505004883, 0.008595088496804237, -1.5729138851165771, -0.9754737019538879, -0.02142678201198578, -0.5195825099945068, 0.5784894824028015, -1.7603031396865845, 0.14198338985443115, -0.9110555052757263, 0.006135798525065184, 0.6464886665344238, 0.271651953458786, -0.7771333456039429, 2.149780511856079, -0.18372486531734467, 1.2636327743530273, 2.4446847438812256, -1.6403051614761353, -2.172278881072998, 0.8131440877914429, 0.5692247748374939, 0.6362081170082092, -0.6238614916801453, -1.1894341707229614, -0.041398800909519196, -0.8114014863967896, -1.563980221748352, -0.7039019465446472, -1.2875890731811523, 0.6358171105384827, 0.2867460250854492, 0.6085441708564758, -0.18670351803302765, -0.035185955464839935, 0.4796321988105774, -0.347789466381073, -0.9608672857284546, -0.2871234714984894, -1.059662103652954, -0.9671469926834106, 0.13815473020076752, -0.8384546637535095, 1.9821423292160034, 0.0034653206821531057, 0.3368889391422272, -0.5542421936988831, -1.3129888772964478, -0.6561866998672485, -1.232366681098938, 0.9192189574241638, -0.5943161845207214, 0.5069791674613953, -2.078312635421753, -1.1215840578079224, -0.003162939101457596, 0.05880226194858551, 0.3570640981197357, 1.4567612409591675, 1.3512440919876099, -1.4632054567337036, 1.1182202100753784, 0.19393496215343475, 2.6846115589141846, 0.07499708980321884, 1.2279096841812134, -0.5861898064613342, 0.05644453689455986, 0.9137008190155029, 0.6584389805793762, -0.7740053534507751, -0.8244331479072571, -0.6914265155792236, -1.5737345218658447, -0.8599390387535095, 1.4339286088943481, 0.40498292446136475, 0.2331748753786087, 0.8175562620162964, 0.6739823222160339, -0.8073192834854126, -0.2074323147535324, 0.23742450773715973, 1.139792799949646, 0.5001189708709717, -1.7582627534866333, 0.12985779345035553, -0.4883537292480469, 1.4900788068771362, -0.5557771921157837, 0.24335767328739166, 1.4627996683120728, 0.12870168685913086, 0.21372821927070618, -0.054332464933395386, 0.9511485695838928, -0.19051924347877502, 1.3295484781265259, -0.10799174755811691, 0.14615756273269653, -0.9282850027084351, 1.4814715385437012, -0.12011615186929703, 0.775005578994751, -1.269105076789856, -0.6755188703536987, 1.5735269784927368, 2.3524909019470215, 0.07628486305475235, 0.2580587565898895, 0.06897670030593872, 0.6830421090126038, -0.9979398250579834, -1.288083791732788, 1.1519701480865479, 0.9981033205986023, 0.46267756819725037, -1.4039751291275024, 0.7731598019599915, 0.5999096035957336, -0.13782146573066711, -0.9490073323249817, 0.614908754825592, -0.4142545759677887, 0.31617259979248047, 0.7169880867004395, -0.3029900789260864, -0.7934487462043762, 0.5921238660812378, 0.3924849331378937, 0.8384835124015808, 1.2189075946807861, 1.5757160186767578, 0.1767417937517166, 2.2399730682373047, 1.7815569639205933, -1.0689305067062378, -0.593370258808136, -0.4103696644306183, -0.5459465384483337, 0.6237124800682068, 0.49220308661460876, 0.005273445509374142, 0.9938327670097351, -1.0096241235733032, -0.07042182236909866, -0.9572738409042358, 0.17404770851135254, -0.38421812653541565, -0.9104822278022766, 0.6901854276657104, -0.022602444514632225, 0.5913947224617004, -0.5579355359077454, 1.3201022148132324, 0.161664679646492, -0.9972590804100037, -0.4087125062942505, -1.7378113269805908, -0.6246351003646851, -1.1773042678833008, -1.439537763595581, -0.2029941976070404, -0.2977239191532135, 0.15758711099624634, 0.6454732418060303, -0.18656450510025024, -0.3272152543067932, 0.12037131190299988, 0.549431324005127, -2.1808884143829346, 0.16242952644824982, -0.2170526683330536, 0.6722760796546936, 1.201796293258667, -0.06868188083171844, -0.4513993561267853, -1.0685197114944458, 0.6757838129997253, 0.07922502607107162, -0.10052754729986191, -0.486917108297348, 0.2382715493440628, 0.09440042078495026, -0.4700702130794525, 0.08958566188812256, 2.0454001426696777, -1.1172688007354736, -0.9243031740188599, 1.8180277347564697, 0.7438079714775085, 0.535158634185791, -0.15679098665714264, 2.1617910861968994, 1.0790139436721802, -0.5668355822563171, 0.3843732178211212, 0.8665086627006531, 1.3660945892333984, 2.6672377586364746, 0.8400600552558899, 0.34250208735466003, -1.8895548582077026, 0.8473886251449585, -0.6831753849983215, -0.592292845249176, -0.7848394513130188, -1.2716020345687866, -0.24012383818626404, -0.5749399662017822, 0.7090638875961304, 1.1037031412124634, 0.29790976643562317, 1.1326777935028076, -0.6337798833847046, 2.4168038368225098, -1.091238021850586, 0.06100619211792946, -0.6063827872276306, -0.6181925535202026, -0.2286679744720459, -0.6888702511787415, -0.3284873962402344, 0.02434656396508217, -2.664658308029175, 0.36571359634399414, 0.489937424659729, -0.38417720794677734, -1.1245465278625488, 1.128525972366333, 0.27630749344825745, 0.4882367253303528, 0.4591810405254364, 0.2681722044944763, -0.019986847415566444, 0.14206445217132568, -0.43925824761390686, 0.3786461651325226, -0.54672771692276, 0.11882252991199493, -0.4091116189956665, 0.6062197089195251, 1.6377168893814087, -0.6873586177825928, 0.4137745797634125, 0.32482749223709106, -2.5401651859283447, 2.0231409072875977, 0.6508292555809021, 1.4822674989700317, 0.434006005525589, 0.354988694190979, -0.29706698656082153, 1.5338066816329956, 0.14113815128803253, -0.9852578639984131, 1.8363569974899292, 0.6128113269805908, -0.2534138560295105, 0.056495510041713715, -0.8224430084228516, -0.9244610667228699, 1.2035913467407227, 0.34509244561195374, -1.9431694746017456, 0.44949692487716675, -0.7018222212791443, -1.9334089756011963, -2.0299346446990967, -1.3355377912521362, 0.9830406904220581, 1.0229883193969727, 0.44740092754364014, -0.5641897916793823, -1.2033746242523193, 0.7698041796684265, -0.6629630923271179, 1.5347983837127686, -0.4394082725048065, -1.2659685611724854, 2.58821702003479, -0.608160674571991, -0.21198321878910065, -1.446723222732544, -0.1864030957221985, -0.11751583963632584, -0.14787061512470245, -0.7030084729194641, -1.6059342622756958, 1.421716332435608, 0.01639401726424694, 0.06178155541419983, 0.13235726952552795, -0.677277147769928, -0.738926351070404, 2.4443461894989014, 0.48974287509918213, -1.5741548538208008, -1.4105242490768433, -0.357757031917572, 1.0851361751556396, 1.6247682571411133, -0.6134706735610962, -1.1843147277832031, -0.15787675976753235, -0.8736198544502258, -1.18968665599823, -0.7157020568847656, -1.2160509824752808, 0.13299690186977386, -2.2017080783843994, -2.2691142559051514, 0.5242446064949036, -1.4261949062347412, -0.3876577317714691, -0.2179993987083435, 1.0974727869033813, -0.08837858587503433, -0.8672339916229248, -1.8972867727279663, 1.1222844123840332, 0.2515536844730377, 0.4472576081752777, -1.109481930732727, -0.09872066229581833, -0.5620708465576172, -0.48934459686279297, -1.287269115447998, -1.328774094581604, -0.1892329901456833, 1.0969970226287842, 0.8586825728416443, -0.32480761408805847, 0.478058785200119, -0.049594733864068985, 1.295109510421753, 0.7808347940444946, 0.18080578744411469, -1.2331194877624512, 0.12355244159698486, -0.1809413731098175, 1.7351624965667725, -0.3505346179008484, -0.21059204638004303, -0.545898973941803, 1.1032811403274536, -0.8935672640800476, 0.3994271159172058, 1.736730933189392, -0.96522057056427, 0.7396964430809021, -0.5733996033668518, 1.556767463684082, -1.8692811727523804, -1.2035480737686157, 0.13333222270011902, -0.25147745013237, 1.2490196228027344, -0.01940593682229519, -0.884244978427887, -1.0253946781158447, -1.0644190311431885, -1.4898998737335205, -0.9704412221908569, -0.5671058893203735, -0.7391395568847656, 0.7111358642578125, -2.311976909637451, -0.07536660879850388, 0.05233310908079147, -0.6832538843154907, 0.7119499444961548, 0.6679677963256836, 1.0118178129196167, 0.04707174748182297, -1.2088514566421509, 0.7704811692237854, -0.03006928414106369, 0.44473227858543396, -0.1441023349761963, -0.7036515474319458, 1.0757697820663452, -1.1590877771377563, 0.8231377601623535, 0.16978514194488525, -0.31132206320762634, 1.273240566253662, 0.8244757652282715, 1.0320329666137695, -0.01659236289560795, -1.8492025136947632, -0.7814180254936218, 0.5839009881019592, 0.10581087321043015, 1.2529683113098145, -1.539481520652771, 0.5783377885818481, -0.40641894936561584, -0.10769365727901459, 0.9116621613502502, 0.8412031531333923, -1.6418728828430176, 0.5275076031684875, -0.24112768471240997, 0.01290868315845728, 0.3774843215942383, 0.3894757330417633, -0.5041569471359253, -1.050691843032837, -0.013610349968075752, -0.7424888014793396, -0.8896085023880005, -1.3190563917160034, -0.2634628415107727, -0.110772505402565, -1.0173829793930054, 0.18373678624629974, 0.3214532434940338, 0.613815426826477, -0.8263413906097412, 1.3731927871704102, -0.8550626039505005, 1.2926266193389893, 0.6625071167945862, -1.626140832901001, 0.9758737087249756, 0.598391056060791, -0.5966468453407288, 0.8228112459182739, -0.6907892227172852, -0.6427924633026123, -0.5310307145118713, 0.7312448024749756, -0.6972835063934326, -0.1298431158065796, -0.08858317136764526, -0.3039705157279968, 1.3942224979400635, 0.28609487414360046, -1.3452364206314087, -1.5713948011398315, -0.176918625831604, -1.2908902168273926, 0.202351912856102, 0.23367749154567719, -1.1853554248809814, -0.2739657759666443, 0.5047549605369568, 1.515834927558899, 0.7706576585769653, 1.6791988611221313, -1.1565738916397095, 0.7889866232872009, -0.14813683927059174, 0.4592979848384857, 1.128579020500183, -0.5756866931915283, 0.2980012595653534, -1.3877507448196411, -1.614179253578186, -0.8475272059440613, -0.6274067759513855, 0.6228976249694824, 1.0078331232070923, -1.2087478637695312, 1.4931443929672241, 1.2503952980041504, 0.4299235939979553, -0.11033641546964645, -0.9913004636764526, -0.510107159614563, -0.3032837510108948, -0.6132867336273193, 0.21182066202163696, -0.7243806719779968, 0.9951374530792236, 0.11499432474374771, -0.4300466775894165, -0.030115030705928802, 0.29468047618865967, -0.004435631912201643, -1.1324084997177124, 0.6294916272163391, 0.15707555413246155, -1.923906922340393, -0.3773878216743469, -0.18078814446926117, -0.5006781816482544, -2.1328744888305664, 1.355509877204895, 1.1115550994873047, -0.32102227210998535, -0.8632192015647888, 0.1576189398765564, -0.19559597969055176, 0.2295776903629303, -0.7411739230155945, -1.4144474267959595, 0.8025531768798828, 1.452156662940979, 0.48960059881210327, -2.278852939605713, -1.3902091979980469, -0.12656757235527039, -0.8440662622451782, 0.5885246396064758, -1.0274772644042969, 0.5090426206588745, 0.4903508424758911, 0.32960230112075806, 0.6451991200447083, 0.0063006035052239895, 0.23647771775722504, -0.5218474268913269, 0.1733550876379013, -0.08501841127872467, -0.27346891164779663, -0.16750924289226532, -0.700618326663971, 1.6215243339538574, 1.5094399452209473, 1.3512405157089233, 0.2201802134513855, -0.32959896326065063, 2.526134490966797, 0.052016690373420715, -1.3063180446624756, 1.5351135730743408, -0.4862021207809448, 0.2766273617744446, -0.0045542228035628796, -1.4790995121002197, 1.0734657049179077, -0.058434829115867615, 0.22646348178386688, -0.35443100333213806, 0.3894071578979492, 1.538632869720459, 1.6673927307128906, 1.656088948249817, 0.024600140750408173, 0.049347177147865295, -2.3858888149261475, 0.9193500876426697, 0.3421981930732727, 1.1153472661972046, 0.39442411065101624, -1.875405192375183, -1.26951265335083, 2.199049472808838, 0.279290109872818, 0.08851158618927002, 0.49915698170661926, 1.7792140245437622, 1.8404556512832642, -0.15503187477588654, -0.6096342206001282, -0.22071576118469238, -0.46463555097579956, 0.014542982913553715, -0.1381930261850357, 0.4155447781085968, 0.2604583501815796, 2.3021953105926514, 0.10773584991693497, -0.5746094584465027, 0.3932236433029175, 0.28654101490974426, -0.8797101378440857, -0.35691845417022705, -1.3073149919509888, -0.0529601164162159, 0.41360723972320557, -0.3385103940963745, 1.5847439765930176, -1.720467448234558, -1.9183945655822754, 0.1255684643983841, -1.2586313486099243, -0.48185086250305176, 0.6798329949378967, -1.21360182762146, -1.9819751977920532, 0.6606101989746094, 0.04732775315642357, -0.9885734915733337, -0.10188981890678406, -0.009071893990039825, 1.3918464183807373, 0.5141505002975464, 2.054945468902588, 1.0107107162475586, 0.3012465834617615, -0.22807982563972473, -0.8307702541351318, 0.7134307622909546, -1.8642730712890625, -0.3129364252090454, 0.7365831136703491, -0.8647127151489258, 0.3149033188819885, -1.3308053016662598, 1.1889499425888062, -1.1210129261016846, -0.6777714490890503, 1.0384894609451294, -0.3435312509536743, -0.6528535485267639, -0.6401678919792175, -0.8565031290054321, 0.15527096390724182, 0.8816550374031067, -0.42365291714668274, -0.3586842715740204, -1.222448706626892, 0.1392664611339569, 0.22952759265899658, -1.0798585414886475, -1.5694879293441772, 1.0310509204864502, 0.16287711262702942, 0.17462091147899628, -0.6582657694816589, -1.2137552499771118, 0.3296661078929901, 1.7873198986053467, -0.7043977975845337, -0.1241002082824707, 0.03343823552131653, -1.8468873500823975, -0.5892017483711243, 0.7951658368110657, 0.6236748695373535, 1.0480318069458008, 0.10127038508653641, -1.6564558744430542, -0.6993644833564758, -0.6888918876647949, -0.08569463342428207, 2.712759017944336, 1.195383071899414, -0.05741169676184654, 1.570424199104309, -0.6534876823425293, 0.1337103247642517, -0.4482799768447876, 1.5868217945098877, 2.214293956756592, 0.7987181544303894, 1.177952527999878, 0.0016366447089239955, -0.07376295328140259, 0.6872943639755249, 0.6175501942634583, 0.12405591458082199, -0.44392865896224976, -1.5491238832473755, 0.8790106177330017, -1.1593396663665771, -0.06699395924806595, -0.08743420243263245, 0.6974982023239136, -0.5050820708274841, -1.2910716533660889, -0.2598770260810852, 0.06137518957257271, 1.4989827871322632, 0.07340778410434723, 0.15168309211730957, 0.4613153040409088, -1.10525643825531, 1.577545166015625, -0.6254504919052124, 0.26478055119514465, -0.568130612373352, 0.150035098195076, -0.7316219806671143, 0.5325657725334167, 0.6066111326217651, 0.07469478994607925, 0.9703183770179749, 0.6174964308738708, -2.5045828819274902, 1.1632821559906006, 0.26571816205978394, -0.814277172088623, -0.2978212237358093, 0.158653125166893], [1.5287196636199951, 0.8316828608512878, 0.13446494936943054, 2.073700428009033, 0.41684043407440186, -1.2371128797531128, 0.7623284459114075, 1.139530062675476, 0.3892529010772705, 1.0856150388717651, 0.8756920099258423, 0.5654599666595459, -0.0030425754375755787, 0.011669928207993507, 0.12612304091453552, -0.45011550188064575, -0.30574217438697815, 0.8665571212768555, -0.7439937591552734, 0.013463946059346199, -1.9962468147277832, -1.1119024753570557, -0.14707230031490326, -0.2978743612766266, -0.0474495068192482, 0.8022289276123047, -0.649475634098053, 0.3332364857196808, 0.5946277379989624, -1.1267316341400146, 0.9600732326507568, -0.5593731999397278, 1.438197374343872, -0.18122896552085876, 0.4863961338996887, 0.3827333152294159, -0.6002551913261414, 0.13674649596214294, 0.7095006704330444, 1.5091947317123413, 1.075014352798462, 0.17708703875541687, 1.1135623455047607, 0.5533452033996582, 0.09320323169231415, 0.9668052196502686, -1.5610103607177734, -1.4731451272964478, 1.153800368309021, -0.22867797315120697, -0.5235514044761658, -0.8139123916625977, -1.2367675304412842, 0.015823833644390106, 0.3166092038154602, 0.674222469329834, -0.5053625702857971, 0.4302133321762085, -0.7794404625892639, 0.2780575454235077, 0.015762999653816223, -0.16850502789020538, -0.8291717171669006, 0.43689942359924316, 0.20384806394577026, -2.2906928062438965, 1.8067874908447266, -0.9366588592529297, -1.8193928003311157, 0.6614820957183838, 0.3393890857696533, 0.26173079013824463, -0.48307862877845764, 0.07673686742782593, -3.1534199714660645, 0.7267575860023499, -0.052279312163591385, 1.1163172721862793, 0.10375496000051498, 0.7420745491981506, -0.49862509965896606, -0.6286098957061768, -0.23694466054439545, 1.8461915254592896, 0.4644753634929657, 1.7941703796386719, -1.4798648357391357, 0.02240716852247715, -0.6587791442871094, 0.19880267977714539, 1.0212020874023438, -0.024945134297013283, 2.03682017326355, 0.44138357043266296, 1.4749988317489624, 1.5563950538635254, 0.720733642578125, 1.0102272033691406, -0.27604052424430847, -1.3131165504455566, 0.8934613466262817, -0.9904366731643677, 0.18078139424324036, -0.7647466659545898, 0.1259281188249588, -0.5924827456474304, -1.2232455015182495, -1.3668274879455566, 0.315723717212677, -2.0091488361358643, -0.5916931629180908, -0.6961409449577332, -0.333607017993927, -0.26574161648750305, 1.3093634843826294, -0.673815906047821, 0.2755405604839325, -1.5443942546844482, 2.5554537773132324, 1.022739052772522, 0.2951081693172455, 0.1964961141347885, 1.5047262907028198, -0.11022593826055527, -0.498691201210022, -0.7692176699638367, -0.4370582401752472, 0.44988828897476196, -0.7933109402656555, -1.7889162302017212, 0.3098324239253998, -0.2727561891078949, 0.39081868529319763, 0.98991459608078, 0.12990421056747437, 0.5600745677947998, 1.1378356218338013, 0.5852414965629578, -0.16130448877811432, -1.5641518831253052, 1.9997854232788086, -0.27524664998054504, -0.9521228075027466, -0.09446576982736588, 0.1597800850868225, -0.9285642504692078, -0.08126328140497208, 0.8051422238349915, 0.7344461679458618, -1.1625292301177979, -0.7168582677841187, 1.3901447057724, 0.24592582881450653, -1.4418870210647583, -0.24820253252983093, 0.4931890666484833, -0.6722187995910645, 2.8107032775878906, 0.6047175526618958, -0.6381250023841858, 0.8766449689865112, 0.604447603225708, 0.06039565056562424, -0.9903382658958435, 0.3469742238521576, 2.2025861740112305, 0.6529721021652222, 1.2191745042800903, -1.6049106121063232, 0.601248025894165, -1.056235432624817, -0.4274851679801941, 2.1264219284057617, -0.5112160444259644, 1.3845309019088745, -1.4578919410705566, -0.8322932720184326, 0.754353940486908, -0.5077057480812073, 0.37976744771003723, 0.8252392411231995, 0.5069540143013, 0.9074013233184814, -0.6311883926391602, 0.07134750485420227, -0.4792615473270416, 0.6176979541778564, -1.3805406093597412, -0.8644120693206787, 0.5951665043830872, 0.11923433095216751, -0.695321261882782, -0.1465446501970291, -0.8494170308113098, -0.4144410789012909, -1.398891806602478, 0.5240022540092468, -0.10976887494325638, -1.059149146080017, -0.2502959966659546, -2.1946394443511963, 1.629972219467163, -0.7019821405410767, -0.6868241429328918, 1.0385119915008545, 1.7243235111236572, 0.5125498175621033, -2.3706722259521484, -1.510094165802002, 0.005119741894304752, -0.43976694345474243, 0.3507637679576874, -0.1547900289297104, -0.9970940351486206, -0.30261462926864624, -1.8794535398483276, 0.21526938676834106, 0.1951531022787094, -0.3588661551475525, 2.1686720848083496, -1.7958762645721436, 1.1740148067474365, 0.4441387951374054, -0.3744671940803528, 1.1662194728851318, -0.9438440799713135, 0.13346926867961884, 0.9367358684539795, 0.4016152322292328, 1.1891504526138306, 0.9871192574501038, -0.5741233825683594, 0.9763696193695068, -1.1148860454559326, -1.875983476638794, -0.18389718234539032, 1.8020286560058594, -0.1121259555220604, -1.3806825876235962, -0.7920568585395813, -0.5077243447303772, -0.0964621976017952, -1.9013792276382446, -0.43284568190574646, 0.1536395251750946, -1.135515809059143, -1.8076132535934448, 0.5883787870407104, -0.7565351724624634, 0.3235216438770294, -0.3344239890575409, 0.15208281576633453, 0.6025835871696472, 0.09186507016420364, 1.9224283695220947, -1.6574296951293945, -1.0651167631149292, 0.6234606504440308, -0.704647421836853, -0.0450187548995018, 0.05633111670613289, -0.024910375475883484, 0.391247421503067, 1.0526628494262695, -0.44081443548202515, -0.20787744224071503, 0.5909857153892517, 0.09717120230197906, 0.5381925702095032, 0.4609767198562622, -1.0808993577957153, -0.7948462963104248, 0.12058056145906448, 0.4171580970287323, -1.080339789390564, 0.6700701713562012, -0.9661311507225037, -0.3924742341041565, -1.8031541109085083, 0.7125873565673828, 1.0210564136505127, 0.8245283961296082, 0.21619351208209991, 0.6074234843254089, -1.41727876663208, -1.2841988801956177, 0.21973995864391327, -1.4601216316223145, -0.5257964134216309, -1.2915358543395996, -0.6581458449363708, -0.1801338493824005, 0.8805726766586304, 1.3489576578140259, -0.5328987836837769, 0.16572856903076172, 0.4055384397506714, 0.43752962350845337, -0.49491143226623535, -0.9952845573425293, 0.6469434499740601, 0.663680374622345, -1.1037436723709106, -2.845078945159912, 1.2096827030181885, 0.8314102292060852, 1.2988486289978027, -1.741689682006836, 0.11323673278093338, 0.37578415870666504, -0.42429280281066895, 0.5831878781318665, -1.2538615465164185, 0.2794426381587982, -0.8297167420387268, 1.1165013313293457, 1.0550209283828735, -2.58736515045166, 0.10455841571092606, -2.0271847248077393, 0.13662078976631165, 0.6902832388877869, -0.38438165187835693, -0.05052350088953972, 0.4041490852832794, -0.5528296828269958, -0.12403842061758041, 1.764087200164795, 0.4664705693721771, 1.0087404251098633, -0.990493655204773, 1.030504584312439, 0.6591305136680603, -1.1763558387756348, -0.37894630432128906, -0.12923893332481384, 0.08482971787452698, -0.4128582179546356, -1.4199951887130737, -0.8480057716369629, 2.3773505687713623, -1.3050695657730103, -0.731998085975647, -0.4249013662338257, 0.5373995900154114, -0.642267107963562, 0.6688135862350464, -1.6492271423339844, -1.1882200241088867, 0.9974605441093445, 0.7322558760643005, 0.4562043249607086, 0.6878663301467896, 2.261993885040283, 0.46254345774650574, -1.119354248046875, 0.4801848828792572, -1.7701466083526611, -0.7874457836151123, 0.20399241149425507, -1.1918591260910034, -0.2610098421573639, 0.12849880754947662, 0.3540489375591278, -0.8226403594017029, 0.8887774348258972, 1.0062928199768066, 0.8347079753875732, 0.08696505427360535, -0.20268799364566803, -0.9461471438407898, -0.2369614690542221, -1.3521926403045654, 1.3399174213409424, 0.3373539447784424, 0.05665042996406555, -0.4919511079788208, 0.8661704063415527, -0.20041589438915253, -1.0568081140518188, -0.3402327299118042, -0.37446141242980957, -0.42868077754974365, -1.4766488075256348, -1.0026659965515137, 0.0861768051981926, 0.4688086211681366, -0.3993191123008728, 0.39919716119766235, -0.13595372438430786, 0.41480743885040283, -1.2788740396499634, 1.701141357421875, -0.6087247729301453, 1.803220272064209, 0.4051508903503418, -1.5321049690246582, 0.5938352346420288, 0.3159068822860718, -0.6948341727256775, -0.5866559743881226, -0.5216464400291443, -0.3615565598011017, 1.1199599504470825, 0.28874456882476807, -0.4518406391143799, 0.7622151970863342, 0.10672007501125336, -0.7828689813613892, 0.03337030112743378, -1.1861673593521118, 1.7389717102050781, 1.2047450542449951, 1.935502290725708, -1.0184823274612427, 0.8588846921920776, -0.8770042657852173, 0.7388893365859985, -0.9967917799949646, 0.4703064262866974, -0.26018258929252625, 1.3551816940307617, 0.7272811532020569, -0.19403351843357086, -0.2999211251735687, 1.4637147188186646, 2.279081344604492, -0.10285556316375732, 1.1555848121643066, -0.18979722261428833, 1.183043360710144, -0.28576865792274475, -0.749350368976593, -0.10296089202165604, 1.0318182706832886, 1.100002408027649, -1.9562307596206665, -1.1926281452178955, -1.2881373167037964, 0.12125448882579803, -0.24215930700302124, 2.3695640563964844, -1.5729295015335083, 0.8105809092521667, 0.651940643787384, 1.5085142850875854, -0.5872344374656677, -1.4404683113098145, -1.2814096212387085, -0.2826947867870331, -1.3025492429733276, 0.3018537163734436, -0.22046831250190735, 0.8404797911643982, -1.206069827079773, -0.13071034848690033, -1.2526134252548218, 1.0915762186050415, 0.39320674538612366, 1.5099835395812988, 0.2952994406223297, -0.10746729373931885, -0.9554017186164856, -0.6776150465011597, 0.5883194208145142, -0.7054536938667297, -0.3879121243953705, -1.4805594682693481, 0.40634721517562866, -1.2341976165771484, -0.2713593542575836, -0.7649837136268616, -0.4725881516933441, 0.7765682339668274, -0.5169717669487, 0.14993058145046234, -0.44915080070495605, 0.6487811803817749, 0.08236347883939743, 2.0675008296966553, 1.691691517829895, -0.6108506917953491, 0.8048487305641174, 0.37424778938293457, 0.7008945941925049, -0.6214693188667297, -0.22923196852207184, 0.837211549282074, -0.007355294656008482, 2.0626773834228516, -0.7599968910217285, -0.9212096333503723, -1.0671970844268799, -1.176960825920105, 0.1802840530872345, -0.4261755347251892, 1.5414172410964966, -0.03913721814751625, -0.7409563064575195, 0.05920177698135376, 2.261380910873413, -1.1777406930923462, 0.24062608182430267, 1.1651092767715454, -0.49287253618240356, 0.5695903897285461, -1.6287602186203003, 0.7424506545066833, 0.529874861240387, -1.3957165479660034, 0.6311393976211548, 0.45038920640945435, 0.1850949376821518, 0.37465792894363403, 0.2895020842552185, 0.5492968559265137, -0.2872757911682129, 0.21939417719841003, 1.4440299272537231, 0.3134458065032959, 0.24399970471858978, -0.05037117004394531, -0.06356798112392426, 1.2942365407943726, 0.3756551444530487, -0.7692140340805054, -1.2818875312805176, 0.5376695990562439, 0.21701154112815857, 0.28889980912208557, -1.496980905532837, -0.469688355922699, -0.6279330849647522, -0.550285279750824, -2.818039655685425, -0.3315591812133789, -0.11338087916374207, -0.65160071849823, 0.37192535400390625, 1.1818426847457886, -1.0334606170654297, 0.04402291029691696, 0.10579890757799149, -0.09103719890117645, -1.1248528957366943, -0.2176203429698944, 0.42450278997421265, 0.6473322510719299, 0.8552902340888977, -0.3318330943584442, -0.28896239399909973, 0.36857253313064575, 0.5912784934043884, 1.0580928325653076, 0.19021965563297272, -0.7304980754852295, -0.06538503617048264, 0.3117218315601349, 0.1755935102701187, -0.09655925631523132, -1.5846048593521118, -1.9290320873260498, -1.729109764099121, 0.6988198161125183, 0.15718527138233185, 0.09205903857946396, -1.906650424003601, 0.21131181716918945, 0.1586223691701889, -1.3187555074691772, -0.2708500325679779, -2.4089739322662354, 0.7081823348999023, -1.1009531021118164, -0.5944279432296753, 1.8250536918640137, -1.007015585899353, 0.9771109223365784, 1.3647079467773438, -0.8655654191970825, 0.7181989550590515, -0.30082064867019653, -2.0537097454071045, 0.27706778049468994, -0.2584575116634369, -0.8657882809638977, 0.8909156918525696, 1.1424150466918945, 0.4968847930431366, -1.350470781326294, -0.87917560338974, -0.10276246815919876, 0.10811139643192291, -0.42362767457962036, 0.5346279144287109, -0.7029852867126465, 0.4112952649593353, 2.4049699306488037, -0.9972237944602966, -0.3322398364543915, 1.738619327545166, 0.3558395504951477, -1.6254909038543701, -1.0856878757476807, -0.46152275800704956, -0.349555104970932, 0.004754485562443733, -0.14936336874961853, -1.466251254081726, -1.3898907899856567, 0.9730195999145508, 0.36913105845451355, -0.49538934230804443, -0.0929470956325531, 0.8828714489936829, 0.20123392343521118, -1.9399563074111938, 0.6970298886299133, 0.15377311408519745, 0.2870071530342102, 0.03552402928471565, -0.408131867647171, -0.11181242018938065, 0.07056058943271637, -1.1090545654296875, 0.9047669172286987, -1.7512069940567017, 0.08588029444217682, -0.5514540672302246, -1.3959959745407104, -0.4806213080883026, 0.35863688588142395, -0.9163675904273987, -0.2923213541507721, -0.7965378761291504, 1.27956223487854, -0.9961016774177551, -0.2697662115097046, 0.5043925642967224, -1.3019763231277466, -1.6238899230957031, 0.011641225777566433, -0.014560065232217312, -1.089654564857483, 1.0594086647033691, 0.748606264591217, -0.10668986290693283, -1.096778392791748, 0.2337753176689148, -0.13872671127319336, 0.17612481117248535, 1.1813620328903198, 0.6658631563186646, 0.5560864806175232, 0.33511149883270264, -0.5695988535881042, -0.7696016430854797, -0.1079924926161766, 0.18873126804828644, 2.907313346862793, 0.4663647711277008, 0.24237339198589325, 0.8658716082572937, -0.23881734907627106, -2.036757230758667, -0.256531685590744, 1.3614975214004517, -0.6195077896118164, 0.04406171292066574, 0.195891335606575, 1.2586308717727661, 1.6176282167434692, 0.8335412740707397, -0.0195096917450428, 0.6993279457092285, -1.0276455879211426, -0.03233010321855545, 1.2519099712371826, -0.02419852837920189, 0.8192206025123596, -3.1339516639709473, -0.6411324143409729, 1.252395749092102, -1.6253294944763184, -1.29637610912323, -0.05487774312496185, 0.592246413230896, -0.04082741588354111, 0.2981559634208679, 1.2975040674209595, 0.00865812785923481, -0.40624168515205383, 1.0083682537078857, -0.6090897917747498, -1.8805662393569946, -0.5406293272972107, 1.2040594816207886, 0.5224584937095642, -0.2542435824871063, -0.31338241696357727, 0.899925947189331, -2.4553487300872803, -0.9876184463500977, 0.3649987578392029, 1.3613076210021973, -0.709092915058136, 0.7070888876914978, -1.1508764028549194, 0.21838922798633575, -0.3925955295562744, 0.4539332985877991, 0.1324433982372284, -1.2946358919143677, -0.9508445262908936, -0.3001341223716736, 0.4294549524784088, 2.1339030265808105, -0.5065510272979736, -0.6356163620948792, -0.15998245775699615, 0.057540979236364365, 0.8470919132232666, 0.20071136951446533, 1.139183521270752, 1.569115161895752, -0.837780237197876, 2.1233410835266113, -1.5462714433670044, -1.107179045677185, -1.7408745288848877, -0.33387240767478943, 0.46609753370285034, -0.329121470451355, 0.05722591280937195, 1.312559962272644, -0.20049193501472473, 1.1627432107925415, -2.444460153579712, 1.6505476236343384, -0.8902479410171509, -2.2665724754333496, -0.8021960258483887, -0.20477978885173798, 0.5480033755302429, 0.9857305288314819, 0.04972565174102783, 0.5739597082138062, -1.2978965044021606, -0.0786779373884201, 0.7655228972434998, -0.2855777144432068, 0.2216462343931198, -0.1645723134279251, -1.0753378868103027, -0.14313726127147675, 0.07674666494131088, 1.637848138809204, -0.19314536452293396, 0.49302199482917786, -0.894140899181366, -1.5312645435333252, 0.2011702060699463, 0.657791018486023, -1.0177700519561768, -0.3038063645362854, 0.6909842491149902, 0.5318098664283752, 0.6276138424873352, -1.0821932554244995, -1.4343435764312744, 1.3227111101150513, -0.5767566561698914, -1.694076418876648, 0.19449946284294128, 0.3369220495223999, -0.6255723237991333, -0.3977602422237396, -0.25196799635887146, -0.6972981095314026, -2.1532814502716064, 0.6904599666595459, -1.5869048833847046, -0.8173511028289795, -0.2457379549741745, 0.11668331921100616, -0.19805178046226501, 0.7731661796569824, 1.4835723638534546, 0.007025623694062233, 1.8411954641342163, 0.0062461127527058125, 0.3851946294307709, -0.8598690629005432, 1.030011534690857, -0.9496155977249146, -0.7015683650970459, 1.0778177976608276, -1.9034230709075928, 1.319240927696228, 0.7662732005119324, 0.4850746691226959, 0.4284519851207733, -2.1086442470550537, -1.6220396757125854, 2.1348114013671875, 0.9932035803794861, -1.3806867599487305, 0.06857374310493469, -0.1989368051290512, 0.6621240377426147, -0.1310650110244751, 0.9006595015525818, -0.7537416219711304, -0.5703696012496948, -0.13251018524169922, -1.0880167484283447, -0.9336409568786621, -0.14779743552207947, -0.417900949716568, -0.5318105816841125, -0.27640339732170105, 0.9686967134475708, 1.1686937808990479, 0.5123382210731506, -0.025198383256793022, -0.9105497002601624, -0.36310243606567383, 0.9151651263237, -0.40556156635284424, -0.26739001274108887, 0.47974738478660583, 0.049979593604803085, 0.41232219338417053, -0.45226573944091797, -0.7766374349594116, 0.4278308153152466, -1.4004299640655518, -1.5291718244552612, 0.22269368171691895, -0.9810192584991455, 0.5957934260368347, 0.20142118632793427, 0.38227182626724243, -0.4716493785381317, 0.022443529218435287, -0.20674633979797363, -0.07972193509340286, 1.0751537084579468, -1.025843620300293, 0.5246332883834839, 0.22722387313842773, -0.47736024856567383, -0.4168495535850525, 1.6147915124893188, -0.4040675759315491, 1.3520655632019043, 0.45392802357673645, 0.9657021164894104, -0.6105737686157227, -0.846520185470581, 0.5676784515380859, -1.1497280597686768, 0.428573340177536, -0.15816982090473175, -0.20487578213214874, -1.0952551364898682, 0.6038467884063721, -1.0029574632644653, 0.6650834679603577, 0.308742880821228, -0.4446210563182831, 0.8995358347892761, -0.20506735146045685, -0.662170946598053, 0.187129408121109, -0.0161493681371212, 0.582722008228302, -0.6019302606582642, 0.38310667872428894, -1.4366118907928467, -0.9341940879821777, 1.873919129371643, -0.7207664251327515, 0.9184769988059998, 1.1403002738952637, -0.2787626087665558, -0.7154545187950134, 0.4949418008327484, 0.38325053453445435, -0.34374672174453735, -0.10048618167638779, -0.48255541920661926, -1.106404185295105, 1.1289317607879639, 0.8923606276512146, 0.6327632069587708, -1.004558801651001, -0.253239244222641, 0.17468194663524628, 0.5115945339202881, -0.6760280728340149, 0.304313987493515, -1.1052953004837036, -0.7163949012756348, 0.47756826877593994, -0.35831528902053833, 1.6641767024993896, 1.7190431356430054, -1.042217493057251, 0.5558144450187683, -0.2946818172931671, -0.3162754476070404, 0.0685482919216156, 0.44459229707717896, -1.075039029121399, -0.4071202576160431, 0.9792007803916931, -0.762616753578186, -1.1845741271972656, -1.041892170906067, 1.095048189163208, -0.05391690135002136, 0.48308780789375305, -0.6856051683425903, 2.75469708442688, 0.13748957216739655, 0.31342244148254395, 0.06821776926517487, 0.6401127576828003, -1.4970916509628296, 0.26504677534103394, -1.0792452096939087, 0.5695117712020874, -0.18125317990779877, -1.4945850372314453, 0.14880409836769104, 0.5040701031684875, -0.8159751296043396, -0.021771904081106186, 1.7427557706832886, -1.0827791690826416, -1.622166395187378, -0.30050212144851685, -0.2354557067155838, 0.6074941754341125, -0.16639737784862518, 1.1530418395996094, 0.6724960803985596, -0.1924523264169693, -1.6616666316986084, -0.05551321431994438, 0.036550652235746384, -0.6075542569160461, -1.4975508451461792, 2.084933280944824, 0.20571830868721008, -0.6683069467544556, -1.9703549146652222, 1.2190709114074707, -1.0173691511154175, -0.4727237820625305, -0.8776196241378784, -1.488792061805725, 2.3457841873168945, -0.5417421460151672, -0.7141795754432678, -0.14274545013904572, 1.6071909666061401, 0.3124622404575348, -0.20825909078121185, 0.5458530187606812, -0.496427059173584, 0.3583657145500183, -0.31156042218208313, 0.3007873594760895, 0.6939288377761841, 1.2735764980316162, -0.6115450263023376, 0.7525578737258911, -0.2820263206958771, 0.6281751990318298, -0.4869287610054016, 1.2354439496994019, 1.36200749874115, 0.4002528786659241, 1.3727829456329346, 1.2286999225616455, -0.07925587892532349, -0.8380512595176697, 0.018917972221970558, -0.7020111083984375, 1.109148621559143, 0.3001457154750824, 0.9148312211036682, 0.7000072002410889, 1.0504391193389893, -0.6209433674812317, -0.09527162462472916, -0.49638423323631287, 0.8285529017448425, -0.41848695278167725, -0.10014045238494873, -0.07422567903995514, 0.5777751803398132, -0.9178486466407776, -0.2137572169303894, -1.072802186012268, -0.8000944256782532, 1.909447193145752, 2.618760347366333, 1.4025517702102661, -1.3215446472167969, -0.07961693406105042, 1.0364868640899658, -0.23710639774799347, 0.31120479106903076, -0.2657213807106018, -0.3621644973754883, 1.7566490173339844, -1.1084603071212769, -0.4487435519695282, -3.0694570541381836, -1.00825834274292, -0.20600950717926025], [1.2529940605163574, -1.3378052711486816, 0.5936787724494934, 0.8784469366073608, -0.4997083842754364, -0.8930096626281738, -1.5932611227035522, 0.3419307470321655, 1.326488971710205, 0.4453461468219757, 0.13392789661884308, 2.5739893913269043, 1.778874158859253, 1.591332197189331, 1.578216314315796, 0.6872930526733398, -0.7399813532829285, 0.7150403261184692, -0.0784386396408081, -2.2172319889068604, 0.021806487813591957, -0.08176707476377487, -0.6568277478218079, -0.026563340798020363, -0.8050534725189209, 0.2946617007255554, 0.029804637655615807, 0.557256281375885, 0.9489519596099854, -0.07300277054309845, 0.5294054746627808, 0.9456264972686768, 1.3607921600341797, -0.6520483493804932, 0.7788218855857849, 0.7261086702346802, 0.35376372933387756, 0.6126063466072083, -2.913064479827881, 0.34178435802459717, 0.1587357521057129, 1.2732659578323364, -0.0012719263322651386, 0.5910234451293945, 0.8973691463470459, -0.1584348976612091, -0.2245076447725296, -0.527853786945343, 0.7836208939552307, 1.487673282623291, -1.1117489337921143, 1.1149555444717407, 0.8377203345298767, 0.5158114433288574, -1.2062398195266724, 1.8228700160980225, 2.8347246646881104, 0.8917186856269836, 0.7732673287391663, -1.8304402828216553, -0.2207338511943817, 0.11353959143161774, 0.8273897171020508, 0.1578977108001709, 1.372763991355896, 0.07831612229347229, -1.237072467803955, -0.22229884564876556, -1.0375858545303345, -0.7081893682479858, 1.0773370265960693, -0.5232618451118469, -1.1728357076644897, 0.07288458198308945, 1.9868848323822021, -0.9655231237411499, 0.25541141629219055, 0.8993297815322876, 0.004797135945409536, -1.0041565895080566, 0.9974788427352905, 0.1380361020565033, -0.2644543945789337, -1.552964687347412, 0.7630206346511841, -0.5608447790145874, 1.8303059339523315, 0.41076862812042236, -0.04261508584022522, 0.4944576025009155, 1.1418112516403198, -0.019955283030867577, 0.5480611324310303, 0.3814188838005066, 0.3956598937511444, 0.8570787906646729, 0.19836105406284332, 0.7374107837677002, 0.6088212728500366, 1.1453801393508911, -0.3132886588573456, -1.211811900138855, -0.35307759046554565, -1.5539225339889526, -0.1331590861082077, 1.013144612312317, -0.07105947285890579, -0.11051235347986221, -0.9298098087310791, 0.09356285631656647, 1.1490440368652344, -0.9189345240592957, 1.2891757488250732, -0.16328322887420654, -0.7257404923439026, 0.49363407492637634, 0.4348843991756439, 0.18215425312519073, 1.5718241930007935, 0.6568391919136047, -0.057781510055065155, -0.9315474629402161, 1.1176044940948486, 0.07238447666168213, -0.42746686935424805, -1.5073477029800415, -1.1983007192611694, 1.7727831602096558, 0.22757363319396973, -0.4507412314414978, -0.8122180700302124, 0.7456804513931274, -0.34356689453125, -0.5347886681556702, 1.2736804485321045, -2.753957986831665, -0.9631991386413574, -0.6485146284103394, 0.16824819147586823, -0.3324611783027649, 0.975634753704071, -0.6290975213050842, 0.04559318721294403, 1.4115928411483765, -1.81801176071167, 0.5433642864227295, 0.3607228398323059, -1.4257316589355469, 0.0730653703212738, -0.11891326308250427, -0.06154395267367363, 1.7039754390716553, 1.306117296218872, 0.11634719371795654, -1.1125982999801636, 1.2214432954788208, -0.6284945607185364, -1.341968059539795, 1.1304030418395996, 0.8797057271003723, 0.5855945348739624, 0.9277604818344116, 0.5483829975128174, 1.1451084613800049, 0.8553887605667114, 0.2836430072784424, -0.04447827488183975, 1.0507408380508423, -0.42386192083358765, 1.3034310340881348, -1.8581600189208984, 1.4698712825775146, 1.8275398015975952, 0.9386902451515198, -1.08317232131958, 0.3624282777309418, -0.7844973802566528, -0.23801647126674652, -1.387441635131836, 1.4059779644012451, 0.260530024766922, 0.2919638454914093, 1.0706813335418701, 0.4944801926612854, -0.7346930503845215, -0.4111337959766388, 1.0866786241531372, -1.2865662574768066, -1.4518173933029175, 0.0509653277695179, 0.4969451427459717, -0.8198131918907166, -0.9444960355758667, -1.8084653615951538, 0.3292571008205414, 0.4316534101963043, -1.4822293519973755, -0.368853896856308, -1.0693855285644531, -1.033113718032837, 0.6501538157463074, -1.350962519645691, -1.2213935852050781, -1.079963207244873, 0.4088270962238312, -0.33282470703125, 1.5462064743041992, -1.058594822883606, -0.2376171201467514, -1.5033024549484253, 0.8501176834106445, -1.3077884912490845, -0.2205929011106491, -1.977663516998291, 1.0203561782836914, 2.187448263168335, -0.870182454586029, -0.00590689480304718, 0.5432167053222656, -0.0908631682395935, -0.1521073430776596, -0.21756389737129211, -1.6618908643722534, -0.10276125371456146, -1.4123709201812744, 1.3356826305389404, -1.1434345245361328, -0.14490170776844025, -1.1381276845932007, -1.442199468612671, -0.7554997801780701, -0.2091756910085678, -1.3841922283172607, 1.687839388847351, 0.2250424176454544, 0.28130313754081726, 0.2879689633846283, 0.992506742477417, 1.2810481786727905, -0.9304476380348206, -1.0289455652236938, 0.5118191838264465, -0.09241153299808502, 1.1832993030548096, -0.6754068732261658, -1.7983452081680298, 0.40863126516342163, -0.7110950350761414, -0.4121743142604828, -0.015508215874433517, 0.5588473081588745, -1.2309293746948242, -0.7560412883758545, -0.30849525332450867, -0.8157355189323425, -1.9229472875595093, -0.14492273330688477, 0.6611643433570862, 0.48285403847694397, -0.6099079847335815, -0.035592902451753616, -0.7447628974914551, -0.486941397190094, -0.68788743019104, -1.6191279888153076, 0.363953173160553, -1.7854931354522705, 0.197018563747406, -0.34801867604255676, 0.7314926385879517, -0.34193870425224304, -1.9158560037612915, -1.4016510248184204, 1.6001858711242676, 1.4022040367126465, 0.4539686441421509, -0.00808529369533062, -0.5665642619132996, 0.18403422832489014, -0.2106625884771347, -0.8950149416923523, -0.8955609202384949, 0.29105135798454285, -0.2665596902370453, -0.027464086189866066, -1.208592176437378, 1.1619046926498413, 2.0030298233032227, -0.6525755524635315, 0.9653898477554321, -2.5498862266540527, -0.1618509441614151, 1.7119544744491577, 1.6820398569107056, 0.39820244908332825, 0.7684486508369446, -0.44998952746391296, -0.16255727410316467, 0.7810022830963135, 0.9937472939491272, 0.15659722685813904, 1.042279839515686, 0.6221288442611694, 0.3624172508716583, 0.3366353213787079, -0.677302896976471, -0.4247772991657257, 0.9719844460487366, 0.7067102789878845, 1.1827136278152466, -0.40573787689208984, 1.046878695487976, 0.585771918296814, -1.5693488121032715, 0.1958576738834381, 0.12712545692920685, 0.7320378422737122, 1.3009811639785767, -0.8970072865486145, 1.2768291234970093, 0.4620615839958191, 0.38941270112991333, -1.544623851776123, 0.6395999789237976, 0.7974994778633118, 1.053607702255249, -0.9580691456794739, -1.009127140045166, -0.5368810892105103, 0.31222838163375854, 0.9865686297416687, 0.6463715434074402, 0.8952694535255432, -0.13358290493488312, 0.6689493656158447, 0.6078857183456421, -1.532829999923706, -0.6037701964378357, -0.38159534335136414, 1.2333204746246338, -0.9409665465354919, -0.8966396450996399, 2.0558278560638428, -0.04433296620845795, 0.5737294554710388, 1.2181739807128906, -1.5270359516143799, -0.3701484203338623, -0.829047441482544, -0.4050401747226715, -0.6595069766044617, 0.9303366541862488, 0.9721753001213074, 0.7819254994392395, -0.39503049850463867, -0.6116086840629578, 0.6867859363555908, 0.6111432909965515, -0.06007416173815727, -0.4959990680217743, 0.19608236849308014, 0.7411686182022095, 0.45113861560821533, 0.026046324521303177, 1.078888177871704, 0.2186274230480194, 0.448764830827713, 0.11551254242658615, -0.6323189735412598, 0.041954781860113144, -1.7581675052642822, -1.8081119060516357, -0.5116365551948547, 3.5029568672180176, -1.5632174015045166, 1.1496615409851074, 0.22492364048957825, -0.12909375131130219, -1.527943730354309, 1.1514042615890503, 0.4627434313297272, -0.8532843589782715, 0.8951205611228943, -0.22484898567199707, 0.8585996627807617, -0.2645138204097748, 1.557571530342102, -0.7894633412361145, 0.8683701753616333, 0.71418297290802, -0.021958863362669945, 2.821932315826416, -1.1188569068908691, 0.11899227648973465, -0.8322746157646179, 0.2514694929122925, 1.2806557416915894, -0.025067754089832306, 0.1673768013715744, -0.03678200766444206, 1.1202178001403809, 0.06809588521718979, 0.04957231506705284, -2.422980308532715, 1.4380983114242554, 1.3703466653823853, -0.06401224434375763, 0.8118887543678284, -1.2096363306045532, -1.0783215761184692, -0.33537811040878296, 0.2616460919380188, -0.6421661972999573, 1.3298512697219849, -0.6180411577224731, 0.8009498119354248, -1.978220820426941, 0.019200675189495087, 0.38998958468437195, -0.7398965954780579, -1.1782252788543701, 0.08484768867492676, 0.46658262610435486, 1.634255051612854, -0.6759337782859802, -1.39920973777771, -0.8617804050445557, -0.17961008846759796, 1.1636466979980469, -0.5729451179504395, -1.791861891746521, -1.548020362854004, 1.2204787731170654, -0.32620203495025635, 0.3114478290081024, 1.4627753496170044, -0.4259895086288452, 0.7852200865745544, 0.37234166264533997, -1.014134168624878, -0.17628233134746552, -0.5021158456802368, -0.939368724822998, -0.5209150910377502, 0.14678546786308289, 0.4732748568058014, 0.6870247721672058, 0.38926947116851807, 0.6836881041526794, -0.3665072023868561, -0.4410839080810547, -0.02626442350447178, -0.036573994904756546, 0.2313997894525528, -0.5733742713928223, -0.27660903334617615, -1.1928447484970093, 0.5537152886390686, 2.1005983352661133, 0.34976962208747864, 0.33656543493270874, -2.263148307800293, -0.7784324288368225, 1.0350149869918823, 0.1848081797361374, 0.05994972586631775, 0.6339641213417053, -0.018764525651931763, 0.5411666035652161, -0.4873272180557251, 0.3913743495941162, 0.9435722231864929, -0.46979913115501404, -0.7982975244522095, 1.8811367750167847, 0.210967555642128, 0.8590429425239563, 1.2222150564193726, -0.7208683490753174, 1.2942962646484375, -0.04676191136240959, 1.4365302324295044, 0.1717720478773117, -0.08109359443187714, -1.3963103294372559, 1.359195590019226, -0.43914103507995605, -0.41866445541381836, -0.8711426854133606, -1.3683595657348633, 0.5751346945762634, 0.3013828694820404, -1.7270947694778442, 0.5089876651763916, -0.9238700270652771, -1.797878623008728, -0.7258208990097046, -0.19842438399791718, 0.8589067459106445, -1.0618226528167725, 1.242722749710083, -2.328638792037964, 1.360337495803833, 0.9011929631233215, -1.170480728149414, -0.7774288654327393, -0.2540186047554016, 1.5240229368209839, 0.20356370508670807, 0.8620011806488037, -0.8078798651695251, -0.5168866515159607, 0.5520326495170593, -0.03296869620680809, 0.232246994972229, 0.7404996156692505, 1.969308614730835, 2.55332612991333, -1.7990427017211914, -0.31797200441360474, 1.8737856149673462, 1.259805679321289, -1.2915621995925903, -1.7503324747085571, -0.338666170835495, -1.4646223783493042, 0.24924099445343018, 0.40988844633102417, 1.2157845497131348, -1.2070109844207764, 0.6587613224983215, -0.035064924508333206, 2.6975231170654297, -1.3261743783950806, -0.05997157096862793, 1.2688218355178833, 0.35107100009918213, -0.9966527819633484, -0.3061065971851349, -0.1377817541360855, 0.6046317219734192, -0.09785673767328262, -0.2672346830368042, 0.162570983171463, 1.216779351234436, -0.573094367980957, 0.12875930964946747, 0.2769075334072113, 0.7714877724647522, 0.975247859954834, 1.1229298114776611, -2.9948983192443848, 1.1315242052078247, -1.1692317724227905, -0.8312340974807739, -0.28537893295288086, -0.3428911566734314, -0.16556794941425323, -0.29235774278640747, 1.422072410583496, 0.7267209887504578, 0.30030369758605957, -0.727083683013916, 1.7406642436981201, 0.8685838580131531, -0.12904347479343414, -0.7996546626091003, -1.2867372035980225, 0.5368790030479431, 0.836858868598938, 1.84770667552948, 0.46143752336502075, 0.6837235689163208, 0.9875050783157349, 1.7189075946807861, 0.5318268537521362, -0.6730953454971313, -1.2141761779785156, -0.4463256895542145, -0.3041081726551056, -0.09752237051725388, -1.202555537223816, -0.3436923921108246, 0.9402578473091125, 0.11917965859174728, -0.2121095508337021, 0.12911199033260345, 0.323030024766922, 0.899318516254425, -0.5149262547492981, 0.5219088792800903, -0.84923255443573, 1.1390695571899414, -0.1611454039812088, 0.28494498133659363, -0.04051773622632027, 1.5974565744400024, -0.7494425773620605, -0.9799054861068726, -0.8971551656723022, 1.5924351215362549, -0.6761614680290222, -1.3967337608337402, -0.7457466125488281, 1.9702576398849487, 0.4482564628124237, 0.22460763156414032, 0.1318904608488083, -0.37763991951942444, -1.2396310567855835, -0.10825848579406738, 0.26302647590637207, -0.015101686120033264, 0.9436578154563904, 0.2501983046531677, -1.684214472770691, 0.039713092148303986, -1.2429207563400269, -0.24768075346946716, -0.9567824006080627, -0.04970164597034454, -0.8756909966468811, 0.6232433319091797, 1.2375801801681519, 0.538213312625885, -0.4493923485279083, 0.2169935256242752, -0.8150690793991089, 0.13739603757858276, 0.21049802005290985, -0.3840511441230774, -0.8724122643470764, 0.4483301043510437, 0.44636571407318115, 0.7496598362922668, 0.023443058133125305, -0.5735430717468262, -0.20907682180404663, -0.5445425510406494, 1.1049453020095825, -0.1821908950805664, 0.32029157876968384, -0.2513793408870697, -0.0013466572854667902, -0.30844882130622864, -0.3108382523059845, 0.49696123600006104, -0.12039434164762497, -0.6442356705665588, 0.613967776298523, -1.8973522186279297, -1.3875854015350342, -0.7571667432785034, 0.42248615622520447, 0.13060443103313446, -0.3062334358692169, -0.6001274585723877, -1.9070948362350464, -0.2023482471704483, -0.40103432536125183, 0.06410244852304459, 1.0637537240982056, 1.0720722675323486, 1.1336036920547485, -0.45213842391967773, -0.0720444843173027, 0.7925207614898682, -0.004130138084292412, 0.6622421145439148, 0.583821713924408, 0.042675573378801346, 1.6361302137374878, -2.859079599380493, 1.041885256767273, 1.5996062755584717, 1.018291711807251, 1.0288939476013184, 1.4233940839767456, -0.23430734872817993, 0.040608327835798264, 1.3296241760253906, -0.39829811453819275, -1.6910665035247803, 1.022804617881775, -0.8454223275184631, 0.35663875937461853, 2.6640563011169434, 0.301240473985672, 1.234142780303955, -0.6323936581611633, -0.4864889681339264, -0.1352906972169876, -0.2519893944263458, 0.45236513018608093, -1.1132043600082397, -0.6751753687858582, 1.3882280588150024, -0.03392095863819122, -0.552644670009613, 0.21595148742198944, 0.5279653668403625, 0.24788165092468262, 1.1758846044540405, 1.5930242538452148, -1.2460671663284302, 0.5752497315406799, -0.6827604174613953, 0.9157671332359314, -1.3816537857055664, -0.03437558561563492, -2.1064555644989014, -0.5124112963676453, -0.6634640097618103, -1.4284954071044922, -0.42331767082214355, 0.6159520149230957, 0.20596015453338623, 0.9213570952415466, 0.9288746118545532, 1.231610894203186, -0.1793278008699417, -0.5114158987998962, -0.742315411567688, 0.5528784990310669, -0.8669965863227844, -0.4570464491844177, -0.24163082242012024, -0.8041703104972839, -1.2039220333099365, 0.5522608757019043, -0.4878818988800049, -1.1847115755081177, -2.2764832973480225, 0.21068502962589264, 0.20031455159187317, -0.1581314355134964, -1.1236134767532349, -0.7742710709571838, -0.2877768576145172, -0.3484790027141571, 1.0991092920303345, -0.13891464471817017, 1.1603871583938599, 1.3957453966140747, -0.21194900572299957, 1.8916312456130981, 0.4518595039844513, -1.0382133722305298, 0.46787530183792114, -0.5291646718978882, 1.890153169631958, -0.36293867230415344, -0.3513215184211731, -0.0005000171950086951, 0.8182603120803833, 0.7327999472618103, -0.6285739541053772, -1.0926510095596313, 0.004670598078519106, 1.5328565835952759, -0.9416292905807495, -0.5415948629379272, -0.14594672620296478, -1.2269014120101929, 0.6890619993209839, 0.23975104093551636, 0.7121587991714478, -1.367684245109558, 0.8898904919624329, 0.9029878973960876, 0.16274888813495636, -0.5541581511497498, -1.374813199043274, 1.3386402130126953, 1.3088665008544922, -1.3916891813278198, -0.7027298212051392, 0.28305429220199585, -1.0886236429214478, 1.0405923128128052, 1.3306740522384644, 1.3053640127182007, 0.2140558809041977, -0.6263319253921509, 0.6638103127479553, 0.2777031660079956, -1.7707189321517944, 1.9707295894622803, -1.9005956649780273, -0.04286588355898857, 1.360098958015442, 0.2569892406463623, -0.19581788778305054, 0.2239151895046234, 0.9642961621284485, -1.1652826070785522, -0.02307489700615406, -0.36331915855407715, -1.1408215761184692, -0.8825562596321106, -0.36631500720977783, -0.750252366065979, 0.5039088726043701, -0.06626352667808533, 0.6643465757369995, -0.06643431633710861, 0.6984708905220032, 1.2936663627624512, -1.6811954975128174, 0.018915850669145584, -0.4623529016971588, 0.6947771310806274, -1.8086227178573608, 0.8443720936775208, 1.2537568807601929, 0.443127304315567, 0.7087527513504028, -0.687126636505127, -0.6710858941078186, 0.062417689710855484, 0.3851626217365265, -0.11297402530908585, 2.218087673187256, 1.1225841045379639, 0.4807327091693878, -0.09812269359827042, 0.9028217196464539, -0.2803107500076294, 0.32019510865211487, -0.35116079449653625, 0.6795847415924072, 0.28971797227859497, 1.0320889949798584, -0.6045971512794495, -2.204502820968628, 0.3306507468223572, 0.41759243607521057, 0.6010915637016296, 0.6691144108772278, -0.2447856068611145, 0.8345105051994324, -1.0282299518585205, 0.01800536923110485, -0.09676351398229599, 0.9886345267295837, -1.268720269203186, 0.6022436618804932, -0.8224186301231384, 1.0233404636383057, 0.8521084189414978, 0.8488485813140869, -0.33526524901390076, -0.7501422166824341, -0.3420634865760803, -0.44930851459503174, 0.19239011406898499, 0.6228732466697693, 0.20547541975975037, 0.24596208333969116, -0.7310298085212708, -0.3874770700931549, 1.3689031600952148, -0.7169721722602844, -1.5627541542053223, 0.621177613735199, 1.0168801546096802, 0.3797917068004608, -1.2681574821472168, -0.6896469593048096, -1.0239548683166504, -0.47833073139190674, -0.14024457335472107, -0.37143129110336304, 0.30053603649139404, 0.11372195929288864, 0.2858368456363678, 1.9104856252670288, -0.21526940166950226, 0.3967978060245514, -1.1338783502578735, 0.6706582903862, -0.664318323135376, 0.03218542039394379, -0.03690696880221367, -0.07556654512882233, -0.10727047175168991, 0.40244054794311523, -1.7000858783721924, -0.614330530166626, -0.8343886137008667, -0.9123328328132629, 1.1566792726516724, -1.27140474319458, -0.0931311622262001, 0.4249589443206787, 1.0251659154891968, -0.20737431943416595, 2.52398943901062, 1.2756494283676147, 0.8681941628456116, -1.0060492753982544, -1.1978970766067505, 2.7743372917175293, -0.17118096351623535, -0.7198454737663269, 2.822983980178833, 0.23149597644805908, -0.15375912189483643, -1.5615661144256592, 1.214414119720459, -0.7261008024215698, -0.577545702457428, 1.3956489562988281, -1.2594190835952759, -0.2607232332229614, -1.1604081392288208, 0.9340535402297974, -1.4152227640151978, 0.5017411112785339, -0.03929256275296211, -0.7491896748542786, 0.5869882106781006, -0.2157372534275055, -0.04153232276439667, -0.22910796105861664, 2.383983850479126, -1.0647813081741333, -0.40401339530944824, -1.3756946325302124, 1.0084415674209595, 1.7109071016311646, 0.28697848320007324, -1.5603808164596558, 0.16245320439338684, 0.7437931299209595, -0.10476371645927429, -0.8835278153419495, 0.9416290521621704, 1.8140451908111572, 0.44389596581459045, -0.23676064610481262, -1.0257011651992798, 0.25329485535621643, 0.6163356304168701, -0.4419007897377014, 1.4926514625549316, 1.5762370824813843, -0.9748458862304688, 0.08781512081623077, 1.3447372913360596, -0.34931281208992004, 1.4401415586471558, 0.4954565763473511, -0.7194784879684448, -0.06023386865854263, 0.5138683915138245, -2.2874937057495117, -1.5502984523773193, 1.2853561639785767, 0.37511029839515686, -1.4137295484542847, 0.314832478761673, -0.5553303956985474, 0.6766489744186401, 0.2756965458393097, 0.02595086395740509, -0.45431631803512573, 1.4297341108322144, 0.842663049697876, 0.9839567542076111, 0.17367078363895416, -0.30577343702316284, -0.27962952852249146, 0.408128947019577, 0.7063930034637451, -1.6780544519424438, -0.8355793356895447, 0.6739278435707092, 0.7878411412239075, -0.9185717701911926, 1.4766826629638672, -0.6756447553634644, 1.3744733333587646, -0.6433485746383667, -2.019268751144409, 0.4173939824104309, -0.7108935713768005, 0.9324077367782593, 0.21901360154151917, -0.6123648285865784, 0.8563447594642639, 0.12292160838842392, 0.8867104053497314, -0.054566625505685806, 1.2642990350723267, 0.0702093243598938, 0.36850327253341675, 1.404373049736023, -0.3464832901954651, 0.06908592581748962, -2.277602434158325, -0.5573711395263672, -1.23054039478302, -0.5046862363815308, -0.10421431064605713, 0.6420931816101074, -0.09344594180583954, -0.04585147276520729, 1.2934496402740479, 1.3454697132110596, 1.6719932556152344, 0.66357421875, 0.4772948920726776, 0.3475123643875122, -0.2210322916507721, -0.17498205602169037, -0.6809551119804382, -0.6401428580284119, -2.150106430053711, -1.5576432943344116, -0.5059977769851685, -1.3350858688354492, 1.1923823356628418, 0.24606476724147797], [-0.6849557757377625, -0.7083238959312439, -0.07743910700082779, -0.9775095582008362, -0.5835781097412109, -1.813835859298706, -0.41215214133262634, 1.365779161453247, -0.29540830850601196, -0.6944097280502319, 0.008021791465580463, 0.7375500798225403, 0.6279866695404053, 0.4372433125972748, 0.16425921022891998, -0.9133156538009644, -1.5793219804763794, 0.6813001036643982, 0.1603017896413803, -1.3520479202270508, 0.06928592175245285, 1.8964009284973145, -0.553363025188446, 1.7808061838150024, -1.1504604816436768, 0.9029925465583801, 0.8734502196311951, -0.04396143555641174, 0.3834502696990967, -1.011194109916687, -1.6171449422836304, -0.7250186800956726, 0.44993308186531067, -0.14481480419635773, 1.0875375270843506, 1.2028419971466064, 0.06028604507446289, -0.2245979905128479, -1.8934215307235718, -1.3089960813522339, 0.7007479071617126, -0.7364937663078308, -0.8913822770118713, -0.28187644481658936, 0.7169049978256226, -1.84615957736969, -0.16733312606811523, 1.138406753540039, -0.005070179235190153, 0.713718831539154, -0.2243361622095108, 0.3979521691799164, -1.0472744703292847, -0.5699495673179626, 1.1645234823226929, 0.43690064549446106, -0.6044946908950806, -1.4823143482208252, 0.9292827248573303, -0.004380564205348492, -1.5734847784042358, -0.23553664982318878, 0.09985921531915665, -0.15515583753585815, 0.13507406413555145, 1.6766184568405151, 0.25774893164634705, 0.8396483659744263, 0.9300464391708374, -0.8170844316482544, 0.39725491404533386, -0.2617366313934326, 0.4581831693649292, -0.4881766438484192, 2.0107922554016113, 2.2290360927581787, 0.3555031418800354, -0.3880784809589386, -1.4775357246398926, 0.10239618271589279, 0.0728289783000946, 1.7968568801879883, 0.40984782576560974, 0.891202986240387, 0.030794279649853706, -0.9652798771858215, -0.8888674378395081, -0.009076638147234917, 0.7255434393882751, 1.1972203254699707, -0.7060937285423279, -0.237168088555336, 0.244470477104187, -1.6598998308181763, 0.16303525865077972, -0.6914193034172058, -0.024384038522839546, -0.16967402398586273, -1.6256595849990845, 1.8513911962509155, 0.6750128865242004, -1.001227855682373, 0.2779829502105713, 1.08358633518219, -1.6198068857192993, 1.8996156454086304, 1.1790589094161987, -0.7387999296188354, 2.0296480655670166, -0.8032737970352173, -0.22106336057186127, -0.042580727487802505, 1.2728372812271118, -0.3561486005783081, -0.03668777272105217, -1.207596778869629, 0.6581342816352844, -0.3980298340320587, 1.181458592414856, 0.16853950917720795, -0.4538581669330597, 1.5334111452102661, 0.27155667543411255, 0.20543353259563446, 0.12433257699012756, 0.488052636384964, 0.8339329361915588, 0.2639520466327667, -0.6397739052772522, 0.2740442454814911, -1.6991468667984009, 0.7565837502479553, 0.2632780969142914, 0.2555617690086365, -0.27255210280418396, 0.18790113925933838, 1.4539198875427246, -0.23602084815502167, 0.3275955617427826, -0.2933434247970581, -1.191540241241455, 0.11113425344228745, 2.405644416809082, 0.7564355134963989, -0.12706422805786133, 0.5314264893531799, -0.4483950436115265, -0.4030454754829407, -1.3361073732376099, -2.379448413848877, 1.3421188592910767, 1.471885323524475, 0.5171080827713013, 0.39906206727027893, -0.3157072961330414, 0.684799313545227, -0.8851149678230286, -0.11244997382164001, -0.8131312727928162, 0.4918772876262665, 0.1637657731771469, 0.021407581865787506, 0.019864099100232124, 0.5699062347412109, 0.18536505103111267, 1.1892566680908203, -0.670932412147522, -0.2901292145252228, -1.3107762336730957, 1.2379957437515259, -0.7890172004699707, -0.6774645447731018, 1.2998017072677612, 1.1330194473266602, 0.25907662510871887, -1.9866482019424438, -0.6786512136459351, 0.474875807762146, 0.4740625321865082, -0.9407998323440552, 1.3335866928100586, 0.4813619554042816, -0.5804914236068726, 0.715114951133728, -0.6051313877105713, -1.512690544128418, 1.0352420806884766, -0.5052107572555542, 0.055116381496191025, 1.7984888553619385, 1.3716200590133667, 0.5866495966911316, 2.281464099884033, -1.903059720993042, -0.7223095297813416, 0.40375030040740967, -1.3387463092803955, -0.12261110544204712, -1.1949217319488525, 0.34552931785583496, -0.2281697392463684, -1.4366768598556519, 0.38067880272865295, -1.4714998006820679, 1.5300674438476562, -1.2115838527679443, 0.23231393098831177, 1.456971287727356, -0.6505126953125, 1.6990103721618652, 0.49261578917503357, 0.24915295839309692, 1.3017231225967407, 0.8249031901359558, 0.1349177211523056, -1.7765486240386963, -1.4180840253829956, -1.6395814418792725, -1.0388637781143188, 1.157360553741455, 0.72042316198349, 0.302993506193161, -0.2715544104576111, 0.47416529059410095, -0.4757942259311676, 0.5580107569694519, -0.41496455669403076, 0.8843072056770325, 1.5004161596298218, 0.4149983525276184, 0.15295758843421936, 0.00740745710209012, 0.5773470401763916, 0.5106943249702454, 0.22623111307621002, -0.8666219711303711, -0.9365017414093018, -0.06526684761047363, -1.5215715169906616, -0.7493020296096802, 1.133394479751587, 0.47263771295547485, -0.13054072856903076, -0.4548385441303253, 0.4262809157371521, 1.2114452123641968, -0.4181145429611206, 0.9091702699661255, -0.9188088774681091, -0.948205292224884, 0.9918356537818909, -0.11212857067584991, 0.965970516204834, 0.23156389594078064, -0.3048456013202667, -1.2349779605865479, 0.7604628205299377, 0.6735071539878845, 0.6895860433578491, 0.31078824400901794, -1.9118399620056152, 1.4264737367630005, -0.9009408950805664, 0.7105048298835754, -0.6048027276992798, 0.24997907876968384, 1.795465350151062, 2.8956127166748047, -0.6662147641181946, -1.2686514854431152, 1.5894529819488525, 0.49943190813064575, -0.10769645124673843, 0.9500188827514648, -0.15326343476772308, -0.21446293592453003, 1.1558222770690918, -2.5244174003601074, 0.8911731243133545, -0.9584997296333313, -1.5481845140457153, 0.2056121528148651, -1.169269323348999, 1.2155039310455322, 0.7471151351928711, 0.5643786787986755, -0.2410198301076889, 1.5616581439971924, -0.1429852545261383, -0.48615583777427673, -0.6178654432296753, -1.2869261503219604, 0.04205694794654846, 0.49182796478271484, 0.789677083492279, 0.9629113078117371, 0.16213302314281464, 1.2471585273742676, 0.22089573740959167, 0.7206209897994995, 0.29643186926841736, 0.581544816493988, 0.3829942047595978, -0.5184274315834045, 2.165100336074829, 0.2750449478626251, -2.4267594814300537, 1.8201329708099365, 0.4027358591556549, -0.5363729000091553, 0.15068718791007996, -0.9440136551856995, -0.08238115906715393, 1.1434005498886108, -1.8919947147369385, -0.976458728313446, 0.24927498400211334, -0.381054550409317, 0.36851850152015686, -0.3977392613887787, -0.8657317161560059, -1.168447732925415, 0.2555682063102722, 0.7672376036643982, -1.1681160926818848, -0.4578038156032562, -1.6008009910583496, 2.481933832168579, 1.2700793743133545, 0.47501447796821594, -0.7283676862716675, 0.32676079869270325, 0.8946839570999146, -1.1193526983261108, 1.002293348312378, -0.15096798539161682, 0.5366782546043396, -0.24271400272846222, 1.3857386112213135, 1.2795500755310059, -0.2749718725681305, -0.03158045560121536, 0.9544193148612976, -0.4396115839481354, 0.24545897543430328, 0.15347987413406372, 0.37775129079818726, -1.8225771188735962, 0.3830903470516205, 0.05254543200135231, -0.06657633930444717, -0.5255895853042603, 0.2973765432834625, -1.8012101650238037, 1.5860791206359863, 0.001905737561173737, 0.34609255194664, 0.6519608497619629, 1.1136713027954102, 0.7072912454605103, -0.4791516661643982, 0.7618255615234375, -3.0034537315368652, -1.8635238409042358, -0.22074909508228302, 1.5797666311264038, -0.22725540399551392, 1.4619566202163696, -0.7826045155525208, 0.3206433355808258, -0.41161423921585083, -0.2091054916381836, -1.044405460357666, -1.8768186569213867, 0.016294127330183983, 0.9936904907226562, 0.9403389692306519, 1.5142756700515747, -0.28976577520370483, -1.8141475915908813, -2.0745420455932617, 0.3157190680503845, 1.1271719932556152, -0.8411040902137756, -0.5511505603790283, -1.4552415609359741, 1.0596277713775635, -0.08081051707267761, -0.5625381469726562, -1.5977492332458496, 0.8807627558708191, 1.6046667098999023, -1.7672969102859497, -0.5189181566238403, -0.8808814287185669, 0.10359451919794083, -0.38149866461753845, -0.4618361294269562, 0.6319657564163208, 2.5488293170928955, -1.804618000984192, 0.6262370347976685, -0.5432583689689636, 0.8631754517555237, 0.7258956432342529, -0.3630387485027313, 0.006389889866113663, 0.9221900105476379, -0.392955482006073, -0.0357392318546772, -0.19421988725662231, 0.5659169554710388, -0.22939537465572357, -1.5333880186080933, -0.2506873607635498, -0.6984661817550659, -0.3717260956764221, 1.0842114686965942, -0.009142878465354443, 0.40688467025756836, 0.4487311542034149, 0.43882879614830017, -1.9032444953918457, -0.24413351714611053, -2.0939443111419678, 0.39186304807662964, -0.40531933307647705, -0.5318761467933655, -1.2386332750320435, -1.758220911026001, 1.2072017192840576, 0.48382997512817383, 0.024728601798415184, 0.48461946845054626, 0.8618055582046509, -0.8699144721031189, 0.8414717316627502, -0.6795336008071899, 0.01784863881766796, 0.9843507409095764, -1.9790551662445068, -0.034328993409872055, 0.1109258159995079, 0.6557044386863708, 0.3199349641799927, -2.098663330078125, -1.484224796295166, -0.9643617272377014, 0.7228792905807495, -0.38874974846839905, -1.0796818733215332, -1.775046706199646, 0.4623998701572418, -0.9882853627204895, -0.7384289503097534, -0.36746782064437866, -1.7139263153076172, -0.42111125588417053, 0.7012104988098145, 0.45647138357162476, 1.1718335151672363, -2.7800772190093994, -1.3362606763839722, 1.0077227354049683, -2.071536064147949, 0.6218600273132324, -0.5912485122680664, -1.9278907775878906, 0.6261425614356995, 1.0379371643066406, 0.12541280686855316, -1.6650208234786987, 0.14633886516094208, -0.16241256892681122, 0.21948549151420593, -0.2993899881839752, -1.0232077836990356, 0.6671661734580994, -0.464356005191803, 0.4411182999610901, 0.47948652505874634, 0.5601550936698914, -0.2088976949453354, -0.9847490787506104, 1.815873384475708, 0.13476283848285675, 1.6210287809371948, -0.9845601916313171, 0.6698219180107117, 0.7046563625335693, -2.500415802001953, -2.267052412033081, 0.41065511107444763, -0.16457419097423553, -0.35347381234169006, -0.576258659362793, 1.7259411811828613, 0.6928455829620361, -0.9207819700241089, 0.8826875686645508, 0.3909480571746826, -1.126817226409912, -0.5246434211730957, 0.9170212745666504, 0.4821522831916809, 0.2099531888961792, -0.21013137698173523, -1.550695776939392, -0.765635073184967, 0.7639806866645813, -1.4568618535995483, -0.7313319444656372, 0.5063443779945374, -0.6515472531318665, -0.08149740844964981, -0.11933272331953049, -1.3241063356399536, -2.530972480773926, 1.7592812776565552, -0.35011598467826843, -0.319876104593277, 0.40290769934654236, -1.0234323740005493, -0.4575148820877075, 0.649898111820221, 0.10308653861284256, 0.5116211771965027, 0.2951800227165222, 0.510394811630249, 0.8693867921829224, 1.246529459953308, 1.9409719705581665, 0.5184608697891235, -0.018866518512368202, 0.6417933106422424, 0.1926080882549286, 0.408463716506958, 0.5101848840713501, -1.120324969291687, 0.6390566229820251, 1.0154225826263428, 0.9032970666885376, 1.1935374736785889, 1.1813733577728271, -0.2903129756450653, 1.3217198848724365, -0.2123299092054367, -0.4614160656929016, -0.24778388440608978, 0.4074679911136627, -0.5602366328239441, 0.09886576980352402, -1.786374568939209, 0.7701134085655212, -0.788816511631012, -0.4620097279548645, 1.9653148651123047, -0.2655825912952423, 1.215778112411499, 0.11591579020023346, 0.416346937417984, -0.5186370015144348, -1.2683992385864258, -1.10384202003479, 0.5939542055130005, 0.6722019910812378, -0.2899704873561859, 0.632044792175293, 0.32094696164131165, -0.2901059687137604, 1.17429780960083, -0.6483049392700195, 0.3843804895877838, -1.568017601966858, -0.5182816982269287, -0.15977653861045837, -1.1599255800247192, 0.9190443754196167, 0.2960635721683502, 1.731203556060791, 0.11964737623929977, 0.2513706386089325, -0.8238922357559204, -1.0499814748764038, 0.09582267701625824, -1.9061146974563599, 0.4785352945327759, -0.2176511436700821, 0.20898275077342987, -0.11885897070169449, -0.45097070932388306, 0.4047006368637085, -2.0545239448547363, 2.0687220096588135, -0.051190249621868134, 0.05965104326605797, 0.7509731650352478, -1.7300851345062256, -0.8767936825752258, -1.012623906135559, 1.186150074005127, -0.37675660848617554, 0.8394465446472168, 0.46371951699256897, -0.14804747700691223, -0.1987515389919281, -0.8077161312103271, -0.5981550216674805, -0.4012789726257324, 1.2838597297668457, -0.731185257434845, 0.14571122825145721, -0.4978000521659851, -0.40713655948638916, 0.04134031757712364, -0.7536574006080627, -0.00954689271748066, -0.9041690230369568, 0.8785663843154907, 2.335901975631714, 0.20604684948921204, -0.35119178891181946, 0.8598577976226807, 0.0799187570810318, -0.15863347053527832, 2.4344496726989746, 0.21302376687526703, -1.1903741359710693, -1.388998031616211, 1.850899577140808, -2.234623432159424, 0.613493800163269, -0.7483530640602112, 1.8554959297180176, -0.24621430039405823, -0.3638631999492645, 0.1525086909532547, -2.3853566646575928, -0.5781857371330261, 0.32438647747039795, 0.6771682500839233, 0.07453860342502594, -0.4104187786579132, 1.5041152238845825, 0.08252856135368347, -0.3041134178638458, -1.2114758491516113, -1.1840928792953491, -1.362935185432434, -0.1522267758846283, -0.6789522767066956, -1.3767729997634888, -0.6327468752861023, -0.42692625522613525, -0.6056824326515198, -2.7166757583618164, 2.3687491416931152, 0.9019664525985718, -1.6986216306686401, -0.02803708240389824, -0.6463447213172913, 0.16130226850509644, -0.02819422073662281, -0.43056562542915344, -0.16336074471473694, -0.4032655358314514, -1.1869282722473145, 0.650312602519989, -1.0003668069839478, 0.010279332287609577, 0.47103527188301086, 2.2156600952148438, -0.1965995877981186, -1.1565083265304565, -0.7870999574661255, -1.6377469301223755, 1.6382828950881958, -0.8224300742149353, -0.8344957828521729, 0.10065266489982605, 0.6834893822669983, 0.8393729329109192, 0.45850786566734314, 0.8742262125015259, 0.005197688937187195, 0.6884312033653259, -1.3098621368408203, 0.04441284015774727, 0.0006621758220717311, -0.3810137212276459, 0.49495363235473633, -0.0405997559428215, 0.6008825302124023, -1.076460599899292, -0.7646604776382446, -1.1563360691070557, -0.3502240478992462, 0.8125063180923462, -1.3565959930419922, 1.3931796550750732, 0.6559603810310364, -0.2487073838710785, 0.15481452643871307, 0.3263492286205292, -0.5089452266693115, -1.241094708442688, -0.38035914301872253, 0.42745888233184814, -0.6027235984802246, 1.4853466749191284, -2.4627115726470947, 0.6638562083244324, -1.13750422000885, -0.5330052971839905, -1.4587821960449219, 0.03389386087656021, -0.3992280066013336, -2.040435552597046, 0.526270866394043, 1.0278475284576416, 1.624741554260254, -1.7835508584976196, -0.1392933577299118, 0.2110847383737564, -0.06146317347884178, 0.973355233669281, -0.07986137270927429, 0.1073664128780365, -0.15082517266273499, -1.7837179899215698, -0.4098653197288513, -0.8414394855499268, -0.09398239105939865, -0.5125407576560974, -0.31495198607444763, 0.2752394378185272, -2.252920150756836, -0.17915062606334686, -1.4964123964309692, 1.0497009754180908, -0.9219858050346375, -1.9121533632278442, -1.8181376457214355, -1.776009440422058, -1.0557571649551392, 0.6210643649101257, -0.8568388223648071, 0.17790000140666962, -0.11611054837703705, -1.6373118162155151, -1.8869812488555908, 0.4218703508377075, -1.2512855529785156, -0.6385932564735413, -0.5141882300376892, 1.7806859016418457, -0.3240272104740143, -1.368889570236206, 0.7337825894355774, -1.0625005960464478, 0.27370038628578186, 0.8711419701576233, 0.93096524477005, 2.0394413471221924, 0.3606710433959961, 0.2460005134344101, -0.9778472781181335, -0.7016127109527588, 0.7977131009101868, 0.03457643464207649, 1.4090780019760132, -0.2779124081134796, 0.06859362125396729, -0.9246941208839417, -0.9076240062713623, -1.4379754066467285, 2.0327298641204834, 1.4143625497817993, -0.8192688226699829, -0.8441495299339294, -0.8046066164970398, 0.6950172781944275, -0.8880597949028015, -1.4398163557052612, 1.5760250091552734, 1.6687229871749878, -0.9483940601348877, -0.49815499782562256, -1.2919422388076782, -1.043817162513733, 0.8378245234489441, 0.328279584646225, 0.22087937593460083, 1.9358655214309692, -1.7556252479553223, -1.314241886138916, -1.9713764190673828, -1.4766905307769775, 0.36696046590805054, 0.3175126016139984, -1.2022954225540161, -0.5104355216026306, -0.4145955741405487, -1.8494609594345093, 1.2940338850021362, 0.09903203696012497, 0.008160455152392387, -0.18366821110248566, 0.1512620598077774, 1.095808744430542, 1.6855889558792114, -0.5892326831817627, -1.4783775806427002, 2.419807195663452, 0.14294910430908203, -0.6584148406982422, 0.6531537771224976, -2.4616997241973877, -0.8290219902992249, -1.4328765869140625, -0.15754666924476624, 0.7194835543632507, -1.1448229551315308, -0.6834188103675842, 0.044927362352609634, 0.329604834318161, -1.3046870231628418, 0.10582977533340454, -1.1332619190216064, -1.3131649494171143, -0.6800037026405334, 0.21129272878170013, 0.36151307821273804, 0.9203917980194092, 1.0876433849334717, -0.6374671459197998, 0.9330871105194092, -0.13187278807163239, 0.6603820323944092, 0.5026453137397766, -0.7698776721954346, -0.6660341620445251, -0.8827175498008728, 0.8109623193740845, 0.1719091385602951, -0.3807263970375061, -1.3530997037887573, -2.274965763092041, -0.7967645525932312, -1.3778612613677979, -0.20353539288043976, -0.5375615954399109, -0.11148394644260406, -0.7388913035392761, 2.247988700866699, 0.43519479036331177, -0.694438636302948, -1.5913509130477905, -0.27441713213920593, 0.616349458694458, 0.13528262078762054, -3.062742233276367, 0.358755886554718, 0.03842354193329811, 0.6798340082168579, 0.42806848883628845, 1.128261923789978, 1.5590581893920898, 0.06856762617826462, -0.49847277998924255, -0.3689144253730774, -1.150107741355896, -0.2836976945400238, 0.16776183247566223, 0.09985842555761337, -0.45446568727493286, 0.2326957881450653, -1.0363606214523315, 2.9762253761291504, 0.21820218861103058, -0.19612440466880798, -1.416520595550537, -0.6843757629394531, -0.11830588430166245, 0.10117538273334503, -0.9945509433746338, -0.6217651963233948, -0.7204681038856506, 0.30931758880615234, -0.5169299840927124, -0.5276479125022888, 0.4193621873855591, 0.11403457820415497, 1.0637755393981934, 0.3581419289112091, -1.154645323753357, -0.10545391589403152, 0.026353003457188606, 0.14118103682994843, -0.6175056099891663, 0.1273469179868698, -0.4788413345813751, -1.0230743885040283, -0.31948360800743103, 0.5668275952339172, 0.7838563919067383, 0.4626401364803314, 0.3867224454879761, -0.41374993324279785, -0.7614001631736755, 0.29867348074913025, -0.31409335136413574, 0.9985401034355164, -0.8791868686676025, 0.02080671489238739, -1.3594077825546265, -1.685699462890625, 0.1238313540816307, -1.9960860013961792, 1.3609596490859985, 1.1240274906158447, 2.402103900909424, 0.4552176594734192, -1.2667824029922485, -0.039019063115119934, -0.9280073046684265, -1.8569964170455933, -2.4266655445098877, -1.1369950771331787, 0.7691351175308228, -1.135664939880371, 1.6928342580795288, -0.34544825553894043, 1.0143784284591675, 0.3559466004371643, -2.121367931365967, 1.9457173347473145, 0.10946023464202881, 0.7891225218772888, -0.7618557214736938, 0.8063828945159912, -0.3988884389400482, -0.2177322506904602, 2.3930001258850098, -0.6688515543937683, -0.22829021513462067, 0.7949804663658142, 0.6927283406257629, 0.7737995982170105, -0.7876259088516235, 0.4071996212005615, -0.5393227934837341, -0.481785386800766, -1.288315773010254, 0.3131534457206726, 0.5422917604446411, -0.3479536473751068, 2.563211441040039, 1.857381820678711, 0.1880866438150406, -0.14317747950553894, -2.293656349182129, -1.8377188444137573, 0.18686793744564056, 0.8934513926506042, 2.051093339920044, -0.04689968004822731, 0.9093022346496582, -1.2144747972488403, -0.22169740498065948, 1.33909010887146, 0.09601467847824097, 0.642118513584137, -1.1307774782180786, 0.1802862286567688, -1.2385395765304565, 0.7160226106643677, 0.7246373295783997, -0.3038444519042969, 0.3176122009754181, 1.2751632928848267, 0.26020702719688416, 0.6177123785018921, 0.8633809685707092, -0.7824157476425171, -1.8094134330749512, -1.468704342842102, 1.3884047269821167, 1.4017049074172974, 1.5205293893814087, -1.1006934642791748, -1.9221534729003906, -0.02152658812701702, 0.01660599745810032, 0.0115120904520154, -0.8806983828544617, -0.1697693169116974, -0.37712034583091736, -0.06981708854436874, -1.0670589208602905, 1.7867450714111328, 0.7728358507156372, -0.05893724039196968, -0.7134714722633362, -0.5752111673355103, 0.11894278228282928, 2.4763145446777344, 0.8313754796981812, 0.9542049765586853, 0.6248895525932312, 0.5400144457817078, -1.1165684461593628, 1.667427659034729, 1.0111709833145142, -0.8401989936828613, 1.327056884765625, 0.7816086411476135, 0.1206604614853859, 0.7060761451721191, -1.3088252544403076, -1.272113561630249, 1.0246111154556274, 1.3432362079620361, 0.8539159893989563], [0.8849391341209412, -1.228987216949463, 1.2504396438598633, -1.653957486152649, 0.6221671104431152, -0.07162365317344666, 0.8976535797119141, 0.03248493745923042, -0.6516908407211304, 0.7559949159622192, 1.2436494827270508, -1.033673644065857, 0.5476459860801697, -0.8455740809440613, 0.5400375127792358, -0.4846256375312805, 1.369134783744812, -1.165705919265747, 0.09976721554994583, 0.8278917670249939, -1.0226538181304932, -1.3071705102920532, 1.0792720317840576, -2.035367012023926, -0.29096856713294983, -0.5541254281997681, 2.2003297805786133, 0.6450457572937012, -1.1305049657821655, 0.7406137585639954, 0.10195028781890869, -2.031794309616089, 0.8240832090377808, -1.5391111373901367, 0.9249958992004395, 1.4136754274368286, 0.20803123712539673, -0.46384716033935547, 0.33391427993774414, 0.055369798094034195, 0.43402984738349915, 0.39720991253852844, 0.5473988652229309, -1.0663114786148071, 1.8963745832443237, -1.0520521402359009, 0.4680309295654297, 0.0027977346908301115, -1.2912911176681519, 0.5902245044708252, 2.1466939449310303, -0.9365534782409668, -1.0788240432739258, -0.3955320417881012, 0.6287416815757751, -0.7079469561576843, 0.46459704637527466, -2.0636892318725586, 1.025097370147705, 0.9800596237182617, 0.09577203541994095, -2.119201421737671, -0.5384681820869446, 0.13114850223064423, -1.2126978635787964, 0.23030292987823486, 1.6872137784957886, -0.7525042295455933, -0.7887347936630249, -1.4686741828918457, -0.67078697681427, 2.220478057861328, 0.35420820116996765, 0.5540157556533813, 1.052122950553894, 0.47958487272262573, -2.413072109222412, -0.09935739636421204, -1.2870551347732544, 0.053049080073833466, -2.7300362586975098, 1.3321800231933594, 0.441983163356781, 1.4336163997650146, -1.3579907417297363, -1.9726321697235107, 0.07210458070039749, -1.3810584545135498, -0.055691204965114594, 0.24168913066387177, -0.9308993816375732, 0.4942432641983032, 0.5663524270057678, 0.28728052973747253, -0.06862655282020569, -1.3162245750427246, -1.0567102432250977, 0.439583957195282, 0.3603617250919342, 0.21856479346752167, -0.8771491050720215, -0.25354915857315063, -0.05832064524292946, -0.5191061496734619, 0.9646784663200378, 0.1613055169582367, -0.27770674228668213, -0.3611234426498413, -0.05556679889559746, -1.1969143152236938, -1.388377070426941, 1.2633671760559082, 1.5987982749938965, -0.3685102164745331, -0.2932093143463135, -0.37348559498786926, 0.5232752561569214, 0.46353524923324585, -1.5657066106796265, 0.7007075548171997, 1.6715543270111084, 2.433655261993408, -0.5405986905097961, 0.6940964460372925, -0.0791621282696724, 0.1672782152891159, 0.940888524055481, 0.21293678879737854, 1.2309200763702393, 0.6222789883613586, -0.12468977272510529, 1.123281717300415, 1.3608462810516357, 2.9414076805114746, -0.6393610239028931, -1.2169147729873657, -1.3419932126998901, -0.2032667100429535, 1.2115949392318726, 0.46926429867744446, 1.1092684268951416, -1.282096266746521, 0.706809401512146, -0.9320364594459534, -0.5886062383651733, 0.20933319628238678, 0.8710370063781738, -0.6196561455726624, -0.026124197989702225, 1.4001469612121582, 0.0753617063164711, 1.3628398180007935, -1.5670706033706665, 1.4612711668014526, 0.1727723628282547, 1.530739426612854, 0.22479233145713806, -1.4727872610092163, -1.8465197086334229, -1.3192107677459717, 1.1942729949951172, 1.3825695514678955, 0.11691655218601227, 0.2554612159729004, -1.2135024070739746, 0.5495409965515137, -0.1726037859916687, -0.2836667001247406, -0.464362770318985, -0.9039681553840637, 1.2381632328033447, -0.5949010848999023, -0.6068893671035767, 0.07514765113592148, 0.17592161893844604, -1.2830649614334106, 0.9886404275894165, 2.1951262950897217, -0.04359742999076843, 0.6781343817710876, -0.6425902843475342, -0.11184261739253998, 0.148516446352005, 0.3970979154109955, 0.4419732689857483, -1.194906234741211, -1.3938379287719727, 0.41936737298965454, -0.16195164620876312, -1.0086842775344849, 0.7936037182807922, 0.37637972831726074, 0.11659349501132965, -1.2080163955688477, -0.8303266167640686, 0.5518555641174316, 0.600152313709259, 0.16447164118289948, -0.45577478408813477, -0.9707959890365601, 0.8802489042282104, -0.8743627667427063, -1.1790822744369507, 0.18676932156085968, 0.820634663105011, -1.5203932523727417, 1.5246381759643555, -0.7033318877220154, -0.6318128705024719, -0.9658567309379578, -0.8790231943130493, 1.9794539213180542, -1.5367329120635986, 0.6422088742256165, 0.7430078983306885, -0.2793673872947693, -0.5250916481018066, -0.39894866943359375, -0.7728990316390991, -2.3560128211975098, -1.5503180027008057, 1.7478647232055664, 0.5790634751319885, 0.9943172931671143, -1.9590673446655273, 0.2680354714393616, 1.4577773809432983, 1.8514134883880615, 0.5365965962409973, 1.0025218725204468, 1.4161818027496338, 0.059717822819948196, -0.04871104285120964, 1.0823824405670166, 0.8152666091918945, -0.844287097454071, -0.17429207265377045, -0.4540705978870392, -1.8811980485916138, 1.1428059339523315, 0.27478909492492676, -0.6622833013534546, -0.8509005308151245, -0.588798999786377, -0.3335689902305603, 0.637470543384552, 0.10011875629425049, 1.415655255317688, -0.6307976245880127, 1.3555068969726562, -1.129002571105957, 0.2929195463657379, -0.40269139409065247, 1.3939648866653442, -0.06571264564990997, 1.704024076461792, -1.3178706169128418, -0.7721163034439087, 0.3053075075149536, 0.9493473768234253, -1.1241514682769775, -1.222960114479065, -1.053553819656372, 0.9489887952804565, -0.5474867224693298, 0.335307776927948, 0.1575227677822113, 0.666002631187439, 0.5373684763908386, -1.794708490371704, 0.07780561596155167, -0.04976244643330574, -0.6708407998085022, 0.28137508034706116, -1.8144012689590454, 0.6112240552902222, 0.6053454875946045, -1.3671772480010986, 1.735024333000183, 1.0478713512420654, 0.5536826848983765, 1.176361322402954, 0.3677332401275635, -1.1492912769317627, -0.3399246633052826, -0.5706144571304321, 0.918890118598938, 0.2957526445388794, 0.8539219498634338, 0.8319838643074036, -1.2959790229797363, -1.1295632123947144, 0.16613984107971191, 0.6900994777679443, 0.4877522885799408, 0.37612709403038025, 0.9794355034828186, -2.0906803607940674, -0.6674197912216187, 0.46435457468032837, -0.3564462959766388, 0.7175329923629761, -0.5330986976623535, -0.5292294025421143, -0.3443491458892822, -1.6182628870010376, 1.2455066442489624, -1.2392832040786743, 1.140600562095642, 0.3740111291408539, -1.1090888977050781, 0.12654119729995728, 1.8932197093963623, -0.8632840514183044, 1.0464869737625122, 1.6685645580291748, 0.6888852715492249, -1.1096076965332031, -0.13946697115898132, 0.2850337028503418, 0.1377967745065689, -0.28505563735961914, 1.0123019218444824, -1.0835834741592407, 0.42064353823661804, 0.32235872745513916, 0.34254252910614014, -0.37863683700561523, -0.36658430099487305, 1.4721828699111938, -0.5083363652229309, -1.088814616203308, 1.13480544090271, 1.381395697593689, -0.4704146087169647, 0.4544798731803894, 1.2594280242919922, -0.19645920395851135, 1.0577538013458252, -0.6525217890739441, 1.0428473949432373, 1.5046560764312744, 1.3427993059158325, 1.377402424812317, -0.3017100393772125, -1.1484578847885132, -0.9741772413253784, 0.567743718624115, 0.434326708316803, 0.0890781432390213, -0.5710254907608032, -0.9220916628837585, 0.9699055552482605, 1.317651391029358, -0.1875637322664261, 2.1648640632629395, -1.10695219039917, -0.5740906596183777, 0.8995142579078674, -0.6785274147987366, -0.1681700497865677, -0.7281593084335327, 0.8148421049118042, -0.9235373735427856, -0.08971403539180756, 0.7701392769813538, 2.0067522525787354, 2.2553443908691406, -0.4021705090999603, -0.3447266221046448, -0.13123103976249695, 2.071688652038574, -0.5468135476112366, 0.9206041097640991, -1.008040189743042, -1.0609300136566162, -1.9420018196105957, -0.027132563292980194, 1.1519629955291748, -2.568458318710327, -0.5750831365585327, 0.13890574872493744, 0.30938711762428284, 0.0655486136674881, -1.7070457935333252, 0.5205758810043335, 0.19737982749938965, -1.702235460281372, -0.8677482008934021, 0.567268967628479, 0.699292778968811, -1.53372061252594, 1.0734093189239502, 0.6266226172447205, -0.9337494373321533, -0.9961402416229248, -1.108862280845642, -0.8618068099021912, -0.8416452407836914, 0.11126279830932617, -0.2524668574333191, 0.856889009475708, 0.010629212483763695, -0.5152611136436462, -0.03939581289887428, -0.32206323742866516, -0.6313881874084473, 0.3162800669670105, 0.0976291075348854, -0.5520747303962708, -0.208001509308815, -0.06651414930820465, -0.7924453616142273, -2.186978816986084, 1.0039654970169067, -0.149151012301445, -0.5365325212478638, -0.09700783342123032, -0.050977930426597595, 0.5380966663360596, -0.5273399949073792, -0.8489609956741333, 0.08458181470632553, -0.3095003068447113, 0.47093769907951355, -1.1099871397018433, -0.35180729627609253, -2.5585763454437256, 0.3057875633239746, -1.9560548067092896, -0.8921070098876953, -1.1928216218948364, -0.32625192403793335, -0.060290079563856125, 0.23690399527549744, -1.1368162631988525, 0.4042972922325134, -0.0727820098400116, 1.2385601997375488, -1.3612899780273438, -0.811662495136261, 0.10369046032428741, -0.6391194462776184, 0.6220914721488953, -0.14961476624011993, 2.677795648574829, -0.3641999065876007, 1.6090452671051025, 0.36942967772483826, 0.3485979735851288, 1.2307661771774292, -1.0847631692886353, 0.9586888551712036, -1.7002317905426025, 0.3224622905254364, 0.09829176217317581, 1.2780288457870483, 0.34162014722824097, -0.7402819991111755, -0.8974490165710449, -0.10131930559873581, -1.8902294635772705, -0.4327521026134491, 1.6285943984985352, 0.7839114665985107, 0.10212461650371552, 0.7658548951148987, -1.2362440824508667, 0.16796976327896118, -0.8912517428398132, 2.2330667972564697, 0.39575791358947754, 0.46143653988838196, -0.6045238971710205, -1.3324006795883179, -1.2028855085372925, -1.167677640914917, -0.871661365032196, 1.032139539718628, 0.13353872299194336, 0.9201906323432922, 0.4971812963485718, 0.006665865425020456, -0.8511364459991455, 1.790818691253662, 0.4298287630081177, 0.00843487773090601, 0.5153892040252686, -0.3364492952823639, -0.6341158747673035, 1.190899133682251, 3.279231548309326, 0.5274696946144104, -0.3366478681564331, -0.4766587018966675, 0.23864421248435974, -0.7958984375, -0.06416066735982895, 0.5390594005584717, 0.2997559905052185, 0.8587282299995422, -0.31296825408935547, 1.742736577987671, 0.023352988064289093, 1.2748156785964966, -0.00879579409956932, -0.0983344241976738, 1.9455355405807495, -0.6121809482574463, -1.7043349742889404, 0.09723523259162903, -0.830899715423584, -0.7340414524078369, 1.1849443912506104, -0.7164983153343201, 1.4730991125106812, -0.9663436412811279, 1.1856460571289062, -0.08592357486486435, 1.2972445487976074, 0.7758572697639465, -1.5181704759597778, 1.3743637800216675, -0.017435956746339798, -0.2120223045349121, -0.7687920331954956, -1.7583562135696411, 0.322533518075943, 0.3216034471988678, -0.05058560520410538, 0.5569807887077332, -0.8089846968650818, -0.5942510962486267, -0.6741448044776917, 1.570576786994934, 0.6626629829406738, 0.6085947751998901, -1.0776817798614502, -0.13609619438648224, 0.9430509805679321, -0.6886123418807983, -0.8154289722442627, -0.4749862551689148, 0.11898979544639587, 0.9128128290176392, 0.00852249562740326, 0.9587275385856628, -1.3069756031036377, -0.20786865055561066, -0.363766610622406, -0.8136731386184692, -0.3135720491409302, -0.4593433737754822, 0.844620943069458, 1.0803977251052856, -0.4984927773475647, 0.3189854025840759, -0.46722427010536194, 0.15543147921562195, 0.08527418226003647, -0.5578272938728333, -0.9164382219314575, -0.06220560520887375, -1.044575572013855, -0.7884041666984558, -0.4680623412132263, 1.0866185426712036, -0.10186328738927841, 0.626406192779541, -0.10682882368564606, -0.21892096102237701, -0.6861671209335327, -0.8650689721107483, -0.6370801329612732, 1.088127613067627, -0.7833284139633179, 3.138108491897583, 0.28877878189086914, -1.7051646709442139, -1.6345125436782837, 1.033921241760254, -2.820375442504883, 1.6360691785812378, 2.224104166030884, -0.9036393165588379, -0.29202941060066223, -0.6422505378723145, 1.0285009145736694, 0.32188913226127625, -0.5195774435997009, 0.712344765663147, -0.7955595254898071, -1.347770094871521, 0.3011537194252014, 0.018244260922074318, -0.2579919993877411, -0.46028634905815125, -1.1588658094406128, 0.22929885983467102, 0.7390629053115845, -0.43826958537101746, -0.06020897254347801, 0.15055759251117706, 0.8338119983673096, 0.07880184799432755, -1.8326451778411865, -0.16760459542274475, -1.0511467456817627, 1.0117700099945068, 2.2162177562713623, 1.057260274887085, 0.8442642688751221, -0.788239061832428, -0.5612751841545105, 1.582802176475525, 0.3555121123790741, -0.21808788180351257, 0.47704145312309265, -0.6619089841842651, 1.2186336517333984, 0.40478551387786865, 0.6174602508544922, -0.24396145343780518, -1.1066358089447021, 0.5416879653930664, 0.241159126162529, 0.2352869063615799, 1.6956409215927124, -0.7574991583824158, -1.93075692653656, 0.4521784484386444, 0.25962382555007935, 2.126232862472534, -0.5995572209358215, -0.7317436337471008, 1.3398109674453735, 0.8217449188232422, -0.5451483726501465, -0.035895757377147675, 1.4385195970535278, 0.2173890471458435, 0.48778221011161804, 1.660028338432312, -1.1920663118362427, 1.0606974363327026, -0.19962771236896515, -0.406238853931427, 0.040915317833423615, 0.44242578744888306, 1.507196068763733, 1.280110478401184, 0.5021547675132751, -0.9395309686660767, 1.5801573991775513, -0.31364116072654724, 0.17765821516513824, -1.7923485040664673, 0.7261847853660583, 1.1342201232910156, 0.23985877633094788, -1.0322442054748535, -0.6073697805404663, -0.7273070216178894, 0.10706734657287598, -0.04803238809108734, 0.3980771005153656, 2.1824281215667725, -0.364210307598114, -1.180643081665039, 1.2518082857131958, 0.38590046763420105, -0.9286348819732666, 0.7944502830505371, -0.24877266585826874, -0.8100021481513977, 0.529524028301239, 0.8293020129203796, -0.06735657155513763, 0.6988210082054138, -1.0105942487716675, -1.3693459033966064, -0.39911213517189026, -1.0834814310073853, 0.07432344555854797, 2.6249570846557617, -0.24863184988498688, -0.4879273474216461, -0.9485200643539429, 0.34131985902786255, 1.1284207105636597, -0.39536720514297485, -0.05523311719298363, -0.08329424262046814, -0.1994531750679016, 1.5477206707000732, -0.525217592716217, -0.27215805649757385, 0.9647526144981384, 1.1266403198242188, -0.9893147945404053, 1.0735394954681396, 0.8858106732368469, -0.3709769546985626, -0.32015690207481384, 0.9596495628356934, 0.9823603630065918, -1.280470371246338, -2.0704710483551025, 0.4476218521595001, 1.1859005689620972, 0.1837247610092163, 0.7384600639343262, 1.4567276239395142, 0.4482384920120239, -1.3815875053405762, 1.1807390451431274, 0.05306244269013405, 1.1182371377944946, 0.01815842092037201, 0.20732487738132477, 0.5099523663520813, 2.0379409790039062, -1.4596598148345947, 0.3532199561595917, -0.29553887248039246, 0.8748601078987122, 0.045173488557338715, 0.8909448981285095, -1.2273151874542236, -1.3930660486221313, 0.5081619024276733, -0.9353708624839783, 0.17797495424747467, -0.03780898079276085, -0.7952020168304443, 1.067699909210205, -0.8328016996383667, -1.4759740829467773, -0.19174408912658691, -0.9749947190284729, 1.3599351644515991, 0.6679320335388184, 0.3541760742664337, 0.0334492102265358, -1.2297941446304321, 1.5895346403121948, 0.5357730984687805, 1.0392627716064453, 0.847678005695343, 0.4708411395549774, 1.8038910627365112, 0.9144869446754456, 1.496394157409668, -0.6106730699539185, -2.1794023513793945, 0.07913784682750702, -0.37590041756629944, 1.0464515686035156, -0.11261166632175446, 0.876373291015625, 2.4207587242126465, -0.24546664953231812, -0.981989324092865, -0.7912645936012268, 0.8827507495880127, -1.0574415922164917, -0.45490896701812744, -1.826297640800476, 0.564346194267273, 1.8025085926055908, -0.6689702272415161, 0.10887004435062408, -1.3515161275863647, -0.27317294478416443, 0.06532374024391174, -0.24039070308208466, 0.534162700176239, -1.1511716842651367, 0.6486760377883911, -0.04714629054069519, -1.0052893161773682, -1.9055814743041992, 0.07897162437438965, -1.3793234825134277, 0.6000592112541199, 0.8117566108703613, -0.963204026222229, -0.04907620698213577, -0.20470799505710602, -0.5917061567306519, -0.41363275051116943, -0.3377673029899597, 0.7121690511703491, 1.1065385341644287, -0.8715535998344421, 0.7517464756965637, 0.07647211104631424, -0.2514496147632599, 0.20873288810253143, 1.807456612586975, -0.2862852215766907, -0.565243661403656, 2.352027654647827, -2.2912063598632812, -1.1545950174331665, 2.551884412765503, 0.25514504313468933, -0.37090203166007996, -0.7273834943771362, 1.240824580192566, -1.2824149131774902, -0.022944219410419464, 1.037718653678894, 2.0640807151794434, 1.1757255792617798, -0.31782370805740356, 1.3621267080307007, -1.0285547971725464, 0.5132433772087097, -1.9560397863388062, -0.6188531517982483, -0.9742940068244934, 0.22928503155708313, 1.5328668355941772, -0.6706300973892212, -0.018069615587592125, 0.4601272940635681, 0.45731908082962036, 0.3113698959350586, 0.19619518518447876, 0.29698646068573, -1.3470079898834229, -0.8874656558036804, -2.3901174068450928, 0.021939530968666077, -1.3715507984161377, -0.5596364736557007, -0.5268909931182861, -0.14379437267780304, -0.20680224895477295, -1.3986499309539795, -0.4021897315979004, 1.729857087135315, 0.5251093506813049, -0.3541926443576813, -0.6323301196098328, -0.009363771416246891, 0.9630960822105408, 0.9309614896774292, 0.3112834095954895, 2.4137909412384033, 0.18992199003696442, -0.6892675757408142, -0.798255980014801, -0.3643105924129486, -0.1858634352684021, 1.4739128351211548, 1.2476156949996948, -1.840133547782898, 0.418649822473526, -0.3384630084037781, 2.083761692047119, -1.5488152503967285, 0.687225341796875, 0.732066810131073, -1.9639613628387451, -2.9237890243530273, 0.20696406066417694, -0.7209802269935608, -0.9943802952766418, 0.08572941273450851, 1.0962573289871216, -0.10768889635801315, -0.02408120594918728, -0.4808383882045746, 0.5203734040260315, 0.10046868771314621, -0.43888914585113525, 0.9062375426292419, -1.0153840780258179, 0.3935093581676483, -1.6818934679031372, 1.8589314222335815, -0.8360437154769897, -1.31830894947052, 0.3625187873840332, 0.01614437997341156, 0.7244927883148193, 1.6348910331726074, 0.930645227432251, 0.08126820623874664, -1.896777868270874, 1.3570947647094727, 0.2755068242549896, -1.8765287399291992, 2.7290728092193604, -0.047280192375183105, 0.6055602431297302, -0.3619122803211212, -0.5043937563896179, -0.4179624617099762, 1.3630437850952148, -1.4882707595825195, 0.7036288976669312, -0.649570643901825, -0.9057286381721497, 1.127259373664856, -0.27914607524871826, -0.11246879398822784, 0.4676956534385681, 2.2816169261932373, -0.6377210021018982, 0.025716710835695267, 0.9336563348770142, 0.6591822504997253, -1.2334572076797485, -0.9592757821083069, -0.011780786328017712, 0.19782587885856628, 0.3928593695163727, -2.4583516120910645, 0.1492965668439865, -1.2659348249435425, -0.08854331821203232, 0.8748013377189636, -0.04018086567521095, -3.3823914527893066, -0.23792624473571777, 0.08704490214586258, 0.0657644271850586, -1.0310981273651123, -2.1541974544525146, -0.9011918902397156, -0.03602141886949539, 1.1840548515319824, -1.361573576927185, -0.19066403806209564, -0.46427905559539795, 0.2882891297340393, -0.36131200194358826, 0.8544193506240845, -1.292664885520935, 2.263151168823242, 1.5178310871124268, 1.1913037300109863, 2.2621254920959473, -0.4226890802383423, -0.5832298994064331, -0.2485181987285614, -1.1205127239227295, -0.290054589509964, 0.7686538696289062, -0.6124857664108276, 0.5067280530929565, -0.8122755289077759, 0.2932299077510834, -0.6836592555046082, 0.41301265358924866, 1.2669262886047363, -0.29360073804855347, -0.44312065839767456, -0.9876992702484131, 0.14115211367607117, 0.9362366795539856, -1.4567242860794067, 0.5999256372451782, -0.3128182888031006, -1.3806155920028687, -0.47114256024360657, 0.7034524083137512, 0.30941057205200195, 0.5937291979789734, -1.147679090499878, 2.557452440261841, 2.478425979614258, -0.07264406979084015, 0.9151705503463745, 0.9609111547470093, -1.2829077243804932, 1.6705020666122437, 0.06402240693569183, 0.8953341841697693, 1.013099193572998, 0.21418166160583496, 0.32433706521987915, -0.9094313383102417, 1.8866996765136719, 0.02216925658285618, 0.3723437786102295, 1.177256464958191, 0.3275562822818756, 0.40552330017089844, -0.4780164659023285, -0.10127267241477966, 0.2119951993227005, -0.7152486443519592, 0.8049659729003906, -0.4883635640144348, -0.5207262635231018, -1.0186076164245605, -0.982045590877533, -0.34380632638931274, -0.5692803263664246, -1.0111840963363647, 0.6475136876106262, -0.6216044425964355, 0.3556677997112274, 1.3505345582962036, -0.5961272716522217, -0.06834358721971512, -0.05648293346166611, -0.8287937641143799, -0.077095165848732, -0.6525878310203552, -0.19575519859790802, -2.123577356338501, 1.4371986389160156, 0.3481713533401489, -1.0045570135116577, -0.49448588490486145, 0.11338012665510178, 0.4230676591396332, -0.6882802248001099, 0.058744776993989944], [-1.7851307392120361, -1.8043577671051025, 0.36636456847190857, 0.3316984176635742, -0.022909756749868393, -0.13651254773139954, 1.8651423454284668, -0.9873091578483582, -1.3960524797439575, 0.4902893900871277, -0.14056333899497986, -0.702969491481781, 1.0772603750228882, -0.7807532548904419, -1.5269697904586792, 0.06366375833749771, -0.30560365319252014, -1.2543184757232666, -2.3341212272644043, 0.8900411128997803, -0.26437532901763916, 1.0909429788589478, -1.0581398010253906, -0.3509945273399353, -1.6268870830535889, 0.7782538533210754, -1.5553839206695557, -0.7334629893302917, -1.5661507844924927, 0.6075235605239868, 1.2030119895935059, -0.3737381100654602, -0.3180747330188751, -0.02099730260670185, 1.1909700632095337, -1.270026445388794, -0.6359413862228394, -1.4238115549087524, -0.32898586988449097, 0.27379757165908813, -2.1964597702026367, 0.9366852641105652, -0.6477273106575012, -0.6870858669281006, 2.0053272247314453, -1.5638889074325562, -0.14925457537174225, 0.5799285769462585, 1.567067265510559, -1.0828076601028442, -0.5177946090698242, -0.5699535012245178, 1.547434687614441, -0.585639238357544, -0.25275757908821106, -0.3462316393852234, -0.245063915848732, 1.551821231842041, -1.3004629611968994, 0.043444421142339706, 0.2302263230085373, -0.846635103225708, -0.5091739892959595, 0.4490019977092743, 1.3135523796081543, 0.47814249992370605, -0.6052321195602417, 0.32330647110939026, 1.267336368560791, -1.5805113315582275, -2.5255725383758545, 0.9885849356651306, -0.7307625412940979, -0.6025509238243103, -0.45036906003952026, -2.2721498012542725, -0.8246058821678162, -0.2241767793893814, -2.0146303176879883, 0.5778886079788208, 2.5003416538238525, -0.9327395558357239, -0.7672544717788696, -0.5312016606330872, -0.2166210412979126, -0.3289886713027954, -0.8650977611541748, 0.8038681745529175, 1.842430830001831, -0.40801042318344116, -2.171393871307373, -0.16006147861480713, -0.8713458776473999, 0.6187970042228699, 1.0026860237121582, -0.8965080380439758, 0.9051962494850159, -2.094619035720825, 0.7523109316825867, -1.324310302734375, 0.20470932126045227, -0.6314908266067505, -0.12752871215343475, -1.5741627216339111, 0.3833672106266022, -1.7800638675689697, 0.43928688764572144, -2.1896018981933594, 0.26987603306770325, 0.8464511036872864, 0.3970029950141907, -1.4985015392303467, -1.7570865154266357, -1.3546127080917358, 0.7098267674446106, 2.829775810241699, -0.1675998866558075, -2.5910708904266357, -0.8718832731246948, 0.7925950288772583, -0.10883103311061859, 0.4851699769496918, -1.0516941547393799, -0.6133263111114502, -1.9503381252288818, -0.009995128028094769, 0.1874919831752777, -2.7986080646514893, 0.40429025888442993, -1.710699439048767, 0.022932447493076324, 1.2342724800109863, 1.2146166563034058, -0.9308344125747681, 0.4923854172229767, -0.5087015628814697, 0.6616231799125671, -1.4878032207489014, -0.3596510589122772, -0.14035764336585999, 0.7342696785926819, -0.1249937042593956, 0.1307477205991745, 0.6611313819885254, -0.16476811468601227, -0.723574697971344, -1.871593713760376, -0.30383726954460144, 0.23982630670070648, -0.35881567001342773, 0.18067900836467743, -1.5323091745376587, 1.145317554473877, -0.2827133536338806, -0.5915538668632507, -1.5263607501983643, -0.4654150605201721, -0.5952979326248169, 2.351591110229492, -0.3611309826374054, 1.1313656568527222, 1.0586172342300415, -0.2593982219696045, 1.592490553855896, 0.2370050847530365, -0.32043030858039856, -0.10096363723278046, 1.0549043416976929, -2.1454038619995117, 0.36863014101982117, -0.15710610151290894, 0.8519523739814758, -0.6148951053619385, -0.40887385606765747, -0.8842567205429077, 1.052051305770874, 0.1711667776107788, -0.1654362827539444, -0.18216972053050995, -0.7037795782089233, -1.5034343004226685, 0.40933850407600403, -1.307562232017517, 0.847082257270813, 0.6483437418937683, 1.60177743434906, 0.33846861124038696, -0.8882138729095459, -2.013483762741089, -1.1886147260665894, 0.16644884645938873, -0.9366472959518433, -1.4908720254898071, 0.2072155922651291, 0.9137436151504517, -0.617144763469696, 0.9639521241188049, 0.9204599261283875, -0.5447073578834534, 0.05579742416739464, -0.8140531182289124, -0.04520276561379433, -0.8491645455360413, -0.667036235332489, 0.5629958510398865, 0.7104412913322449, 0.32554271817207336, 0.20976105332374573, 1.1729650497436523, 0.4020339548587799, -0.05651428923010826, 0.3054136037826538, 0.9226018786430359, -1.39300537109375, 1.120493769645691, 0.1865609586238861, -0.9910018444061279, -0.688289999961853, 0.5192221999168396, 0.8193153142929077, 0.3995773494243622, 0.1354643702507019, -0.7094811201095581, 0.6508467197418213, -1.6807541847229004, 0.6686147451400757, -0.8985484838485718, -1.1978271007537842, 0.01163219753652811, 1.2088384628295898, 0.21912090480327606, -0.7125723958015442, -1.0782160758972168, -0.4798072874546051, 0.3028416037559509, -0.49433308839797974, 0.7430334091186523, 0.15633748471736908, 1.3227864503860474, -1.4832227230072021, 0.07628422230482101, 0.8968780040740967, -0.8681119084358215, -1.5669609308242798, -0.35416916012763977, 1.5260562896728516, -0.7296177744865417, -1.075405240058899, -0.9903827905654907, -1.744170904159546, -0.8767164945602417, 0.12463067471981049, -0.777593731880188, -1.6323531866073608, -0.097327820956707, -0.8557302355766296, -1.3657888174057007, 0.3735131025314331, -0.0685088112950325, 2.0036349296569824, -0.5235686302185059, -1.229914665222168, 0.6173685789108276, 1.0671218633651733, 0.8636751174926758, 0.797549843788147, -0.3816634714603424, 0.24107539653778076, 0.049107301980257034, 0.5988653898239136, -0.03995731472969055, -0.17432647943496704, -0.715168297290802, 0.30158987641334534, 0.18703804910182953, 0.7950513362884521, -1.45332932472229, 0.9581602811813354, 0.3676292300224304, 0.6828803420066833, 0.1916351318359375, -1.5349111557006836, 1.209039568901062, -1.0838278532028198, -0.4466090500354767, -0.5828936100006104, 1.2397971153259277, 1.3427557945251465, -0.02330547384917736, 0.4455345869064331, -0.18808332085609436, -1.8086230754852295, -0.4189561903476715, 0.06420660018920898, 0.3120364844799042, -0.21862420439720154, 0.5744073390960693, 0.3633273243904114, -1.2560906410217285, -0.4924343228340149, 0.3884265720844269, -0.07755092531442642, -1.175978422164917, 1.0324119329452515, -1.457108736038208, -0.7007455229759216, 0.7739951014518738, -0.47514283657073975, -0.4090304374694824, -0.1327713131904602, -0.6023099422454834, -0.13294202089309692, -1.2144417762756348, -0.3344283699989319, 0.316535085439682, 0.6982129216194153, 0.32303887605667114, -1.2241886854171753, -0.06574450433254242, -0.28190532326698303, 2.073758840560913, 0.05135653540492058, 1.2792117595672607, -0.12154338508844376, -0.07870391756296158, 0.48848775029182434, 2.142524480819702, -1.7457733154296875, -0.681079626083374, -0.7923574447631836, -0.27365636825561523, 0.5652053952217102, -2.190042734146118, -0.2307044267654419, -0.10378558188676834, 0.01773369312286377, 0.13434931635856628, 0.0802106037735939, 0.7916572093963623, 1.1851942539215088, 1.1607630252838135, -1.0835121870040894, -0.01060633547604084, 0.7286285161972046, 1.1357176303863525, -0.2081589698791504, 1.6927058696746826, 0.9283740520477295, 0.29539018869400024, 0.7690627574920654, 0.7060050964355469, 0.5835360288619995, 0.8240692019462585, 0.6626173257827759, -0.233979731798172, 2.875235080718994, -0.1663885861635208, -1.0738935470581055, -0.08591850101947784, 1.7120884656906128, 0.32612866163253784, -0.05708266794681549, -0.03511541336774826, 1.820486307144165, -2.187574863433838, 0.7433358430862427, -0.9477736353874207, 1.6100174188613892, 1.6920925378799438, -1.3913159370422363, -1.0952624082565308, 0.4315124750137329, -0.2716704308986664, 0.35303694009780884, 0.8161410093307495, -1.060349464416504, -0.08936811238527298, -0.09904627501964569, -1.8594900369644165, 1.0524588823318481, -0.17935486137866974, 1.1222842931747437, 0.9057379364967346, -0.5901288390159607, 0.4209662079811096, 0.49982014298439026, -0.21383027732372284, -1.1129778623580933, 0.18656538426876068, -1.421775460243225, 0.056251734495162964, -0.2594110667705536, 0.34049859642982483, 0.9988371729850769, 0.8877730369567871, -1.262102484703064, 0.0660167708992958, -1.9846999645233154, 1.6851838827133179, 1.0155811309814453, -1.870710015296936, -0.14797188341617584, -0.1149216741323471, 0.04704258218407631, 0.07657530903816223, -0.6904097199440002, 1.485634207725525, -0.8162223696708679, -0.053397100418806076, 1.4289458990097046, -0.2613842487335205, -0.32104676961898804, 2.3598618507385254, -0.0003044889308512211, 0.15205839276313782, 1.0401405096054077, 0.32094094157218933, -0.2536827027797699, -1.3133704662322998, 0.39292627573013306, 0.768175482749939, 1.8519728183746338, 1.3372795581817627, -0.7508909106254578, -0.8004335761070251, -1.713272213935852, -1.4966566562652588, 0.7233284711837769, -0.8245046138763428, -1.2595665454864502, 1.1668022871017456, 1.610966682434082, -0.8345141410827637, 1.0898687839508057, -0.46969425678253174, 0.6968784332275391, -0.6565953493118286, 0.6712768077850342, -0.4002765715122223, 0.2633001208305359, 0.12367650121450424, -2.205603837966919, -0.7184843420982361, -1.0238537788391113, -0.5687854290008545, 0.5310988426208496, 1.2045819759368896, 0.2496050000190735, 1.2803601026535034, -1.2227721214294434, 1.6006959676742554, 0.050966907292604446, -0.7952631711959839, -0.6650934815406799, -0.3567546010017395, -1.096704125404358, -0.1324128955602646, 1.2314319610595703, -0.30714309215545654, -0.648122251033783, 2.0612449645996094, 0.668847918510437, 0.9547435641288757, 0.6172362565994263, 1.0506443977355957, 0.6956411004066467, -0.21873371303081512, -0.5017394423484802, 0.5762453079223633, -0.730811357498169, 0.24371977150440216, -1.6017065048217773, 0.7214343547821045, 0.8746272921562195, -0.5788010358810425, -0.22129125893115997, 0.12114962190389633, -2.0060830116271973, 0.23073039948940277, -1.3325551748275757, 0.06928977370262146, 0.20286794006824493, -1.763001799583435, 0.23254412412643433, 1.2078033685684204, 0.5989065766334534, -1.3010640144348145, -0.3990282118320465, -0.6858558058738708, 0.49890732765197754, 0.6053863763809204, -2.363053560256958, -1.5538755655288696, 0.42808622121810913, 0.49709585309028625, 2.226715326309204, -0.697252094745636, -0.5807689428329468, 0.2151179015636444, 1.1740856170654297, -0.7851434350013733, -1.1312826871871948, 0.8420278429985046, 0.5275569558143616, 0.23028048872947693, 1.1614569425582886, -1.5339598655700684, 0.35682541131973267, 0.9740958213806152, 1.131085991859436, -0.9616323709487915, -1.2025506496429443, -0.6252764463424683, -0.6796620488166809, -2.5071043968200684, 0.021378498524427414, -1.0092910528182983, -1.0874128341674805, 0.43739065527915955, -0.2915131151676178, 0.513066828250885, 1.0782383680343628, -0.8999741077423096, -1.4241470098495483, -0.6343027949333191, 0.6255489587783813, 0.19758041203022003, -0.6433475017547607, 0.4796072542667389, -1.3951199054718018, 0.7161663174629211, -0.550507128238678, -1.5838050842285156, -0.7884224057197571, -1.4798433780670166, 1.4149929285049438, 0.3059139847755432, 0.06815192103385925, 2.3040528297424316, -2.3754634857177734, 0.1799941062927246, 0.9477330446243286, 1.425532341003418, -0.6299486756324768, 0.5851204991340637, -2.1131484508514404, 0.9821146130561829, -1.973022699356079, 0.2360767275094986, -1.24368155002594, 1.1149500608444214, 1.7406623363494873, 1.2918999195098877, -0.5473424792289734, 0.8027659058570862, -0.8554375767707825, -2.890732526779175, -0.771644115447998, 0.7365522384643555, -2.074221611022949, -0.06351970136165619, -0.39863455295562744, -1.3366668224334717, -0.6299315690994263, 0.042095452547073364, -1.367345929145813, 0.7123410701751709, 1.103388786315918, -1.6988025903701782, -1.6161304712295532, 0.4798668622970581, -0.03870705887675285, -0.4456959664821625, -0.43314051628112793, 0.09952262789011002, 1.5833224058151245, 0.19598236680030823, 0.17375242710113525, -1.351514458656311, 0.7770322561264038, 0.1719042956829071, -0.017939239740371704, 0.3156229853630066, -0.8835664987564087, 0.7422633767127991, 0.7699437141418457, 1.592309594154358, -0.11756747961044312, -2.415682315826416, 1.1446869373321533, -1.4017257690429688, 0.7090781331062317, 0.5601362586021423, 1.053813099861145, 0.8518981337547302, -0.2972411811351776, -0.45996400713920593, -1.09088933467865, 0.27093568444252014, 0.7013451457023621, 0.5028095841407776, -0.21329841017723083, 1.361891508102417, 1.160545825958252, -1.3472318649291992, 2.5885744094848633, -0.017740555107593536, 2.0330488681793213, -1.8835912942886353, -0.026702478528022766, -0.5879195332527161, -0.28107619285583496, 0.7868824601173401, 2.118729591369629, 0.19512972235679626, 0.26280519366264343, -1.0490251779556274, 0.3260434567928314, 2.2558536529541016, 0.4849354028701782, -0.06984592229127884, -1.0980955362319946, 0.7928824424743652, -0.25415605306625366, -0.9180788397789001, 0.7642883658409119, -0.011732031591236591, 0.8667899966239929, 0.9064910411834717, 0.7909963726997375, 0.34506362676620483, -0.8887631893157959, -0.6228371858596802, -0.06796231865882874, 1.0632648468017578, -0.29889094829559326, -1.2263033390045166, 1.9256060123443604, 0.47887247800827026, -1.5354372262954712, -0.5537881851196289, 0.9973781704902649, 0.3017316460609436, -1.2016009092330933, 0.14871971309185028, 0.014871793799102306, 1.336649775505066, -1.140434980392456, 0.2293775975704193, 0.9459933638572693, -0.7259912490844727, 0.39937451481819153, 0.04976026341319084, 0.5445610284805298, -0.8977833986282349, -0.20262357592582703, 0.22568687796592712, 0.045567918568849564, -0.6767449378967285, 2.2768561840057373, -0.8208953142166138, 0.9402338266372681, -1.0896520614624023, 1.038556456565857, -1.1427141427993774, -0.11188584566116333, 1.3587521314620972, -1.201945424079895, -0.6168789863586426, 0.8378660678863525, -2.118029832839966, 1.6870523691177368, -0.5984053611755371, 1.9067450761795044, -0.4326862394809723, 1.5147343873977661, -0.6383177638053894, -0.261385977268219, 0.24025118350982666, -1.4516289234161377, 0.3179490864276886, -0.9220908880233765, 2.2683610916137695, -0.18986746668815613, 1.284704327583313, -0.3162493407726288, -0.5146251320838928, 0.7544216513633728, 1.0063894987106323, 1.9818238019943237, 0.7242466807365417, 0.29699137806892395, 0.45845142006874084, 0.12725679576396942, 1.8838127851486206, 1.5872409343719482, 0.28503966331481934, -1.435080885887146, -0.08486777544021606, -1.6117455959320068, 0.26352226734161377, -2.1648435592651367, -2.862410068511963, 0.14589086174964905, 0.07770900428295135, 0.9079620242118835, -2.338224411010742, -0.6776437759399414, 0.5868315100669861, 0.5095992088317871, 1.8622462749481201, -2.522702932357788, -1.4770317077636719, -1.3720102310180664, -0.48495039343833923, 0.4398753046989441, 0.14506608247756958, -0.11732480674982071, -0.6465688347816467, 1.387338399887085, 1.448489785194397, 0.23496659100055695, 0.009490625001490116, -1.128984808921814, 0.7181673645973206, -0.26372766494750977, 0.6561833620071411, -0.35330578684806824, 0.9167773723602295, 0.08153610676527023, -1.1274598836898804, 0.9161299467086792, -1.3535213470458984, -0.5293130278587341, -2.1119544506073, 1.3634133338928223, -1.3345918655395508, 1.5865159034729004, -0.4679110646247864, -1.0156084299087524, 0.22108064591884613, -0.5520322918891907, -0.4142272174358368, -0.5400734543800354, -0.3726671040058136, 0.8491495847702026, -0.9914961457252502, 0.9440121650695801, -0.39846593141555786, -0.3944832980632782, -0.535639226436615, -0.188406839966774, -0.030660035088658333, -1.0864697694778442, -1.7403488159179688, -0.6052528023719788, -0.20637263357639313, -0.7793103456497192, -0.1804385930299759, -1.5623193979263306, -1.3139296770095825, 0.6896798610687256, 1.124413013458252, -0.08047830313444138, -0.13741572201251984, 1.8587141036987305, -0.9852591156959534, -0.5091502666473389, 1.003385305404663, 0.3127483129501343, 0.9869616031646729, -1.519858717918396, -0.08183667808771133, 1.3097894191741943, 0.7785630822181702, -0.8028735518455505, 1.193900227546692, 1.6984267234802246, -0.3400767147541046, -1.77146577835083, -1.590496301651001, 0.39155641198158264, 0.18930162489414215, 1.0168890953063965, 0.4517335295677185, 1.2107900381088257, 0.6124829053878784, -0.2670021653175354, -1.392429232597351, -0.3560376465320587, 0.45125916600227356, -1.7555853128433228, 2.0120387077331543, -0.5727483630180359, 0.06089123710989952, 0.19131286442279816, -1.3626744747161865, 0.23973682522773743, 0.19148460030555725, 0.056223466992378235, 0.07261316478252411, -0.6458249092102051, -0.1554277092218399, -0.16757246851921082, 0.061301540583372116, -1.4536759853363037, -0.5463581085205078, 0.6040805578231812, -0.4388073682785034, -0.42745548486709595, -0.21160489320755005, 0.060314152389764786, -0.8007231950759888, -0.8299195766448975, 0.9299110174179077, -2.70147442817688, -0.22289320826530457, -1.36027193069458, -0.2846527397632599, -0.9705860614776611, 0.11527669429779053, -1.0913430452346802, -0.7197802066802979, -0.10154461115598679, 0.3674508333206177, -1.1815321445465088, -1.4617336988449097, -1.548928141593933, 0.3037623167037964, -1.2704272270202637, -0.6195587515830994, 0.12363880127668381, -0.03603332117199898, -1.0356171131134033, 0.7540633082389832, -0.7890912294387817, 0.730155348777771, 2.541194200515747, -1.0671064853668213, -0.43949565291404724, 1.7153542041778564, -0.23894570767879486, -0.6039074063301086, -0.1321175992488861, -2.2934818267822266, 1.2906911373138428, 0.8857761025428772, 1.5027742385864258, 1.2108049392700195, 0.10695352405309677, -1.9115283489227295, 0.9712845087051392, 2.0495357513427734, 1.1879256963729858, 0.22771893441677094, -0.5723596811294556, 0.22896312177181244, 0.2030022144317627, -0.9769469499588013, 0.5500710606575012, -0.6873067021369934, -0.7022104859352112, -0.22577981650829315, 1.156697392463684, 1.0686479806900024, -0.30282920598983765, -0.8106330037117004, -0.5272639989852905, -0.3999171257019043, 0.39258310198783875, -2.1597683429718018, -0.08377351611852646, 0.6202937960624695, 0.510922908782959, 1.16978919506073, -0.20769056677818298, 0.17550189793109894, 0.7014744281768799, -1.0899980068206787, 0.4523221552371979, 0.8999326229095459, -0.027268029749393463, -0.40582889318466187, -0.9153879880905151, 0.040174420922994614, 0.874160647392273, 1.560360312461853, -0.12026137858629227, 2.289335012435913, -1.2418349981307983, -0.18587176501750946, 0.557639479637146, -1.033900260925293, -1.4809287786483765, -2.3496384620666504, 1.0361136198043823, 0.7831662893295288, 0.5237907767295837, -0.7601567506790161, 0.6602675318717957, -1.1825100183486938, -1.4417985677719116, -0.8891041278839111, -0.15000249445438385, -2.0041747093200684, 0.05100210756063461, 0.8713440895080566, 0.8652939796447754, 0.010166816413402557, 0.9720866084098816, 0.5471753478050232, -0.647436797618866, -0.5654009580612183, 1.074089527130127, 1.4984841346740723, 0.15737523138523102, -1.248997688293457, 0.00797234196215868, 0.5710234642028809, 0.5478902459144592, -1.7175860404968262, -1.6395519971847534, 0.3994278013706207, -1.2383527755737305, 1.3477835655212402, -0.34028077125549316, -0.570801854133606, 0.8420806527137756, 2.6222684383392334, -0.2118779718875885, -1.0600651502609253, 0.16430583596229553, -0.4204525053501129, -0.9795277118682861, -0.3114820718765259, -1.3212591409683228, 0.3278495669364929, 1.3211073875427246, -1.1016175746917725, 0.33561456203460693, 1.11024010181427, 0.9232343435287476, 1.1522961854934692, -0.20488393306732178, 0.6422252058982849, 0.35705873370170593, 0.8355010747909546, 1.6139105558395386, -1.1895415782928467, 2.9285824298858643, 1.7905117273330688, 0.0895741879940033, 0.09205418080091476, 0.3813648819923401, -1.1510875225067139, 1.4137635231018066, -0.9527426958084106, 0.889035701751709, -1.9228368997573853, -1.2997007369995117, 0.07148287445306778, -0.4810411334037781, 0.9333938360214233, -0.6319859623908997, 1.8184317350387573, -0.2461165189743042, -1.2530543804168701, -1.8975105285644531, 2.410728693008423, 0.032642170786857605, -0.020010732114315033, 0.8557189106941223, -0.3192853629589081, 0.006456291768699884, -1.175425410270691, -0.03630024939775467, -0.4537179470062256, 1.357466220855713, -0.1546684205532074, 0.5663909912109375, -0.8407279253005981, 2.515072822570801, -1.2160587310791016, -0.9283888936042786, -0.668430745601654, 0.34047821164131165, 2.439157724380493, 1.0580841302871704, -0.9590599536895752, 0.17593683302402496, -1.0424937009811401, -0.17425115406513214, 0.9980384707450867, 0.5562376379966736, -1.3985401391983032, -0.8278104662895203, -1.1999081373214722, -0.39233922958374023, 0.2639510929584503, 1.598387598991394, 0.729322075843811, 0.789812445640564, -0.05632108822464943, -0.25838059186935425, -0.5624107718467712, 1.147350788116455, 1.4291669130325317, -0.27099230885505676, 0.23249664902687073, 0.4819084107875824, 1.2615138292312622, -1.3642792701721191, -1.5125067234039307, 1.3545914888381958, -0.5749099850654602, -1.931898593902588, -0.1866695135831833, 0.45894500613212585, 0.07723642885684967, -1.859157681465149, 0.4670293927192688, -2.3082873821258545, -0.20130111277103424, -0.24211381375789642], [0.39222705364227295, 0.013178995810449123, -1.3845549821853638, -0.4574185311794281, -1.113417387008667, -0.4499560296535492, -2.062715530395508, -0.632709801197052, 0.42300358414649963, 1.171235203742981, -0.483206570148468, 0.0914754644036293, -0.6685463786125183, -0.08079215884208679, 2.1428589820861816, -0.10292797535657883, 1.5894092321395874, -1.1430753469467163, 0.32033082842826843, -0.6004645228385925, -0.8001824617385864, -0.4087908864021301, 1.0122478008270264, -1.6882256269454956, 0.6367908120155334, -1.00453782081604, -0.881624162197113, -0.7295534014701843, 1.0988205671310425, 0.20794421434402466, 0.7704394459724426, -0.07035110890865326, -0.9176568984985352, 0.4217967987060547, 0.44177091121673584, 0.23110200464725494, 0.7498897314071655, 0.8799901604652405, -0.29441168904304504, 1.4746867418289185, 0.2804955542087555, -0.1391313374042511, 0.5020400285720825, 1.8603096008300781, -1.0103398561477661, 0.5431167483329773, 0.13652803003787994, -0.5712391138076782, 0.5544828772544861, 0.6129070520401001, 2.3959922790527344, 1.808955192565918, -0.3966834545135498, 1.2100898027420044, -2.9860453605651855, 1.9790369272232056, 2.103026866912842, 1.6902960538864136, -0.1473437398672104, -1.7524462938308716, 0.8177260756492615, 0.5732430219650269, 0.2602691948413849, -0.042802438139915466, 0.4385327398777008, -0.9380888938903809, 0.9083074331283569, 0.22331750392913818, -0.9599207043647766, 1.5799086093902588, 0.10164389759302139, 0.4114622175693512, 0.2569422125816345, -0.021584313362836838, 0.5853796005249023, 1.6286882162094116, 1.8648452758789062, -0.12640157341957092, 0.11939205229282379, 0.544471800327301, -0.041661906987428665, 0.0513545386493206, -1.9622604846954346, -1.1788784265518188, 1.3640975952148438, 0.5035747289657593, -0.612079381942749, 1.9187097549438477, 0.3216102421283722, -1.941198468208313, 0.8604279160499573, -0.18576256930828094, -1.094336748123169, -0.26964226365089417, -0.12990568578243256, 0.05297675356268883, 0.752983033657074, 0.5798695087432861, 1.3528039455413818, 1.6402443647384644, -0.6933549046516418, -1.2751898765563965, -0.2094641625881195, -0.2798093855381012, 0.13383139669895172, -1.6776649951934814, -0.5044820308685303, 1.6152262687683105, 0.9970595240592957, 0.2766374349594116, -0.8786008954048157, -1.4903925657272339, 0.26567980647087097, 0.25677359104156494, 0.7523882985115051, -0.8963556885719299, 0.8118748068809509, -0.27611979842185974, -0.1557135283946991, -0.5999942421913147, 1.0861709117889404, 1.5898594856262207, 0.9057701826095581, -1.3583482503890991, -2.212407112121582, 1.6031601428985596, -0.09518945962190628, 0.2153220772743225, 0.30184534192085266, 0.7257553935050964, 0.8257445693016052, -0.05504891276359558, -1.7397561073303223, -0.32086893916130066, -0.06622026860713959, -1.0906997919082642, 2.5620737075805664, -0.6169501543045044, 1.9620627164840698, 0.34956225752830505, 0.1786411702632904, 1.1883257627487183, 0.1899895817041397, 1.350291132926941, 1.309268593788147, -0.15868066251277924, -0.34769561886787415, 0.6292147040367126, 0.3994767963886261, 0.6021808385848999, -0.25660887360572815, -0.18817263841629028, 0.36489903926849365, -1.4306334257125854, -1.479063630104065, -0.4890654385089874, -0.21032361686229706, -2.407681941986084, -0.75673508644104, -0.27594438195228577, -1.0605870485305786, 0.40364402532577515, -1.6164391040802002, -1.7484267950057983, -1.347308874130249, 0.7498311996459961, -0.5054468512535095, -0.18124690651893616, -1.2197861671447754, 1.5193222761154175, 0.9937848448753357, 0.1862611323595047, 0.23403576016426086, 0.18920722603797913, -0.1392131745815277, 1.4215121269226074, 0.16915519535541534, 0.701045036315918, 0.5156753659248352, 0.4023813009262085, 0.6001622676849365, 0.31713297963142395, 0.762608528137207, 0.9506765007972717, 2.7635631561279297, 1.1792583465576172, 0.3856239318847656, -0.5362458825111389, 0.7806011438369751, -1.0971715450286865, 0.1436307728290558, -0.5652722716331482, -1.3643850088119507, -1.148518681526184, 1.0686264038085938, -0.9463390111923218, 1.3668581247329712, 0.9974213242530823, -0.7094196081161499, 1.161817193031311, -0.19910503923892975, 1.0022379159927368, 2.120715379714966, 0.12067466974258423, -0.7949952483177185, -1.3987406492233276, -0.13283373415470123, 1.4691880941390991, 1.677567958831787, 0.04947207495570183, 0.64311283826828, -1.0488958358764648, 0.4712856113910675, -1.0989930629730225, 1.175531268119812, -0.34584471583366394, -0.8816182613372803, 0.6804463267326355, -1.3692433834075928, 0.5655269026756287, 0.09281039237976074, -0.7610839009284973, -0.7324680089950562, -0.14853616058826447, 0.7161226272583008, -0.704719603061676, -0.45787060260772705, 2.013490915298462, -1.0664372444152832, -0.8869312405586243, -0.034699395298957825, -0.7280465364456177, -1.4942235946655273, -1.075722575187683, 0.8719249963760376, 0.8021053671836853, 1.606580138206482, 0.43343448638916016, -0.7714011073112488, 0.1575668603181839, 0.5316555500030518, -0.2464764416217804, -1.7379939556121826, -1.5614548921585083, -0.8475833535194397, -0.09384553134441376, -0.7202465534210205, 1.648910641670227, -0.610644519329071, -0.10978566110134125, 0.4437316656112671, 0.8341356515884399, 0.7379549741744995, 1.191115140914917, 0.9288964867591858, 0.9905811548233032, -2.5013904571533203, 2.1954495906829834, -0.23463237285614014, 1.4991505146026611, -0.8906366229057312, 2.217395782470703, 0.8859072327613831, -1.1285789012908936, 0.6895749568939209, 0.6256251931190491, 0.45275384187698364, 0.0937657579779625, 0.7160569429397583, 0.7565088868141174, -0.9115187525749207, -0.9388489127159119, 0.2647106349468231, -0.17794111371040344, 0.10921760648488998, -0.9154025316238403, 0.27897632122039795, -1.3332746028900146, 1.2358484268188477, -1.405686378479004, 0.9708395600318909, 0.2228846698999405, 1.5326734781265259, -1.3587290048599243, -0.1881498098373413, 0.35637518763542175, -0.9904358386993408, 0.632032573223114, 0.8613935708999634, 1.2606587409973145, 2.0870344638824463, 0.8723758459091187, -0.07332605868577957, -0.483305424451828, -0.7049257159233093, -1.895745038986206, 0.415569543838501, -1.322228193283081, 0.5923241376876831, -1.5166279077529907, 1.1897928714752197, -0.4389283359050751, 1.0375256538391113, 1.4510982036590576, -0.7703502178192139, 0.8213303685188293, 0.057770781219005585, 1.2877541780471802, -0.5932247042655945, 1.0724598169326782, -0.37181809544563293, 0.5689274668693542, 2.3595924377441406, -0.6459358334541321, -1.3169703483581543, 0.6085136532783508, -2.2398579120635986, 1.026013970375061, -1.3312894105911255, 1.6828750371932983, -0.029491066932678223, 0.44017669558525085, -0.018826311454176903, 1.655472993850708, -0.09069021791219711, 0.1376960575580597, 2.005256414413452, 0.9872704744338989, -2.4268641471862793, 0.8188945055007935, 2.5208892822265625, 0.766660213470459, -0.7363018989562988, -0.8017908930778503, 0.47260358929634094, 0.6909722089767456, -1.4903721809387207, 0.31047409772872925, 0.5754590034484863, 0.1837955266237259, -1.3745123147964478, 0.5048821568489075, -1.4017022848129272, 1.3149235248565674, -0.16431449353694916, -0.9841490983963013, -0.8408524990081787, -1.5833081007003784, 1.4797612428665161, 1.3570681810379028, 1.549206018447876, 0.47636550664901733, 0.8335731029510498, -0.15128996968269348, 1.3252601623535156, -0.19253569841384888, 1.1106821298599243, -1.7871228456497192, 1.1639899015426636, 0.2014860063791275, -0.5895131230354309, 0.8196706175804138, 0.6159878373146057, -1.664578914642334, -1.3883135318756104, 0.18795141577720642, -1.2841492891311646, -0.25842252373695374, 1.43400239944458, -0.07079468667507172, 1.2291719913482666, -0.15036413073539734, -0.15409915149211884, -1.7183822393417358, -0.269301176071167, 0.1625489443540573, 0.5500420928001404, 1.4378191232681274, 1.1010479927062988, 2.0636801719665527, -1.2079565525054932, 0.11108140647411346, -0.20968404412269592, -0.4498192071914673, 1.2494242191314697, 1.4544814825057983, -1.1725095510482788, 1.1148524284362793, -1.0322728157043457, -1.1058107614517212, -0.10817789286375046, 0.6877509951591492, -1.2811890840530396, 1.2880847454071045, 1.1116669178009033, 0.1292603760957718, 1.9510486125946045, -0.37681207060813904, -0.4122108221054077, 0.25989216566085815, 1.70832359790802, 1.4093434810638428, 0.7829451560974121, -0.5083168745040894, -1.982771396636963, 0.6792726516723633, 0.48336103558540344, -1.6819344758987427, 0.18185670673847198, 1.768588662147522, -0.30423739552497864, -0.3994474411010742, 0.039195600897073746, -1.1817806959152222, -0.10611642897129059, -0.8762706518173218, 0.7460247874259949, 0.6706141829490662, 2.814438581466675, -0.36488309502601624, 0.625316858291626, -1.0733129978179932, -1.1459522247314453, 0.19213366508483887, 0.9958100318908691, 1.0539124011993408, 0.5936251282691956, -1.2750591039657593, 1.138263463973999, -0.1575305312871933, -0.6778364777565002, -0.80660080909729, -0.4439159631729126, 0.9591429829597473, -0.45917168259620667, -1.3022700548171997, 1.4719057083129883, -0.41291430592536926, 1.8825886249542236, -1.5280029773712158, 1.053800106048584, -0.08477181196212769, 0.2064405083656311, 1.7070101499557495, -0.3780183494091034, 0.21444915235042572, 0.7640884518623352, 0.058043044060468674, 0.4373880624771118, -1.0612225532531738, -1.382401943206787, -0.9111047983169556, 0.5742881298065186, -0.672745943069458, 0.37264060974121094, -0.9085628986358643, -0.4864087402820587, -0.8849694728851318, 1.2447503805160522, 0.2907581031322479, -0.22875739634037018, -0.26866087317466736, -1.3013638257980347, 0.7720602750778198, 0.8515633344650269, 0.19126030802726746, 0.5058555006980896, -1.0362340211868286, -1.3972223997116089, 0.564842164516449, 0.014875324442982674, 0.17848962545394897, 0.40086591243743896, 1.2376830577850342, 0.3808634281158447, 0.7198023796081543, -0.621882975101471, -1.2436808347702026, -0.14633876085281372, -1.553663969039917, -0.9864291548728943, -0.8880537152290344, 0.10734854638576508, 1.3172616958618164, 1.3652969598770142, 2.1786272525787354, -0.5781973600387573, 0.41286221146583557, 0.605366051197052, -0.2238440066576004, -1.2914600372314453, -0.8335570693016052, -1.2931127548217773, -0.4122611880302429, 0.624790370464325, -0.4938518702983856, 1.1447175741195679, -1.604276418685913, -0.369123637676239, -0.15404535830020905, -0.25221797823905945, -0.03016829863190651, 0.7982679605484009, 0.5606261491775513, 1.18704092502594, 0.8317033052444458, 1.2036250829696655, 0.6857694387435913, -1.0606578588485718, -0.965116560459137, 0.7568247318267822, 1.7020823955535889, -0.46680307388305664, 0.05969788879156113, 0.9859614968299866, -0.17470654845237732, -0.15269270539283752, 2.9672765731811523, 1.0118426084518433, -0.05657651647925377, -1.916395902633667, 1.6784217357635498, 0.3926519453525543, -0.6151159405708313, -1.6647614240646362, 0.207306370139122, 1.5579514503479004, -0.5450016856193542, 0.6543103456497192, -1.604048252105713, -1.1655621528625488, -0.19172115623950958, -0.31236451864242554, -0.4527500867843628, -1.0741429328918457, -0.749764084815979, 0.5687156915664673, -0.8537564873695374, -0.43867406249046326, 0.1923849880695343, -0.8400787711143494, 1.2032779455184937, 0.6102688312530518, -0.18305537104606628, 0.25261369347572327, 0.9221325516700745, -1.3830058574676514, 0.06084972992539406, -0.9056240916252136, -2.364942789077759, 0.029707390815019608, -0.993571400642395, -2.253082752227783, 0.5212817192077637, -0.9255408644676208, 0.9171371459960938, 0.10259590297937393, -0.6027708053588867, -0.5836561322212219, -0.5032951235771179, 0.2930412292480469, -0.22944536805152893, 0.4947700798511505, -0.5471170544624329, -0.8128468990325928, 0.8166163563728333, 0.18454816937446594, 0.16106444597244263, -0.45717310905456543, 0.10765408724546432, 0.7163813710212708, 0.04984012991189957, -1.018066644668579, -0.5136698484420776, 0.3019760549068451, 0.48547840118408203, -0.7419155240058899, 0.8574201464653015, 0.28769609332084656, -0.669738233089447, -0.6173762083053589, 1.460541844367981, 1.3233208656311035, -1.6778208017349243, 1.7547763586044312, -0.26741907000541687, 0.9076120853424072, -0.3617147207260132, 1.3358845710754395, 0.8449049592018127, -0.4429788589477539, -0.2522370517253876, -0.15183494985103607, 0.09306271374225616, 1.220807433128357, -0.08603687584400177, -0.5182812213897705, 0.38839489221572876, -0.8222173452377319, -0.11108991503715515, -0.3177523910999298, -0.38641151785850525, -1.745163083076477, 1.5518033504486084, -0.12239768356084824, 0.24765223264694214, 0.0942072793841362, 0.09764590114355087, 0.3183755874633789, -1.8331533670425415, -0.35681790113449097, 0.11281496286392212, 0.8722743391990662, -0.1292160153388977, -0.9232155084609985, -1.7038726806640625, 1.272801399230957, -0.18332898616790771, 0.5934866070747375, -1.2255802154541016, -1.0080506801605225, -1.2893356084823608, -0.39995405077934265, 0.017509371042251587, -0.38319283723831177, 0.9335452914237976, -1.4115086793899536, -0.09841793030500412, 0.00177849386818707, 0.1088770404458046, 0.25331977009773254, 1.266334056854248, 0.658424973487854, -0.5009152293205261, 0.9406481981277466, 0.999298095703125, -0.494976282119751, 0.7583401799201965, 0.5277447700500488, 1.5637420415878296, -0.9238592386245728, -0.5589442849159241, -1.0889759063720703, 0.6065030694007874, -0.922262966632843, -0.37226200103759766, 0.638621985912323, -1.1290318965911865, -0.640706479549408, 0.48424842953681946, 0.606447160243988, -1.0382380485534668, 1.290551781654358, -0.03843523561954498, 0.7645366191864014, -0.4938168227672577, -1.6495245695114136, -0.17657123506069183, -1.0747486352920532, 0.5409634709358215, -1.708372712135315, 0.2543672025203705, 1.2071460485458374, -0.24654315412044525, -0.6351879835128784, 0.052316877990961075, -1.2336552143096924, -1.5150479078292847, 1.0970518589019775, -1.3297418355941772, 0.2834338843822479, 0.08681841939687729, -0.5415346026420593, -1.9996832609176636, -1.2641257047653198, -2.0845072269439697, 1.137257695198059, 0.38076817989349365, -0.5337154269218445, -1.020327091217041, 0.6825592517852783, 1.1711727380752563, 0.17395009100437164, 0.12160380184650421, -0.30063462257385254, 0.4898896813392639, -1.5225098133087158, 0.4922940135002136, 0.1079447790980339, -0.8735294342041016, -0.5974862575531006, 0.5132559537887573, 0.6337628960609436, 1.136650800704956, 0.5430590510368347, 1.418360948562622, -0.3331659138202667, 0.2754925489425659, 1.5475016832351685, 1.0033420324325562, -0.9504302144050598, 0.8269915580749512, 1.1726840734481812, -0.8301490545272827, -1.2497776746749878, -0.9830125570297241, -0.016064761206507683, -0.7424018979072571, -2.2332592010498047, 0.48141708970069885, -1.51048743724823, 0.8569892048835754, 0.5130984783172607, -0.5955081582069397, -1.4537073373794556, -0.3563171327114105, 0.4182502329349518, 0.6738694906234741, -0.30714133381843567, -0.5975140333175659, -0.6078245043754578, -0.7240314483642578, -0.8720529675483704, -0.1310683637857437, 0.6884134411811829, -0.7364671230316162, -0.4807361960411072, 0.18130147457122803, -0.14679789543151855, 0.4015871584415436, 0.7009223103523254, 1.2184251546859741, -0.4501034617424011, -1.2317878007888794, -1.1319692134857178, -0.3700927793979645, 1.7852758169174194, -0.5329302549362183, -0.8543564677238464, -1.8619635105133057, 0.6080518364906311, -0.14594025909900665, 0.9819747805595398, -1.6644630432128906, 0.46235090494155884, 0.11064064502716064, -0.3327590525150299, 0.06291405111551285, 0.27006995677948, 1.2835803031921387, -1.0181999206542969, -0.05636531487107277, -0.8693888783454895, -1.409749984741211, -0.6974018812179565, -0.3979625105857849, -1.0687329769134521, 1.0124319791793823, -0.36635661125183105, 0.6822845935821533, -1.1779152154922485, -2.08402419090271, -0.7606052756309509, -0.14933185279369354, -0.6026619672775269, -0.39690154790878296, -0.3782975375652313, 1.7084360122680664, -0.2384830266237259, -2.0326974391937256, -1.4443773031234741, -0.669919490814209, 1.1426573991775513, -2.12959623336792, -1.2935419082641602, -0.5867925882339478, 1.8107731342315674, -0.38311776518821716, -1.6160330772399902, 0.6897560358047485, 1.1045640707015991, -0.019471056759357452, -0.1453942507505417, -0.988337516784668, 0.24653668701648712, 0.22647976875305176, 1.2194172143936157, 0.15938885509967804, 2.1723546981811523, 0.6562323570251465, 2.317495107650757, -0.051440104842185974, 1.4372079372406006, -0.3529774248600006, 0.6502026915550232, -2.276057720184326, -0.22023215889930725, -1.6152706146240234, 1.0979408025741577, 0.11096598953008652, -0.9770933389663696, -0.07107985019683838, 0.16604112088680267, -0.6815747022628784, 2.087634801864624, -0.44157424569129944, -1.007092833518982, 0.4621642827987671, 0.25991520285606384, 0.5531640648841858, -0.17977681756019592, 0.46777403354644775, -0.8146235346794128, -0.7247148752212524, 1.5659680366516113, -0.7985787987709045, -0.3505096137523651, 0.45065245032310486, 0.29285386204719543, 0.3579499423503876, 1.019750952720642, 0.9218035340309143, 0.46659427881240845, 1.5262867212295532, 1.8255285024642944, 0.16479165852069855, 0.021047884598374367, -1.5995593070983887, -1.5841693878173828, -0.5253379344940186, -0.973533570766449, -0.5221831798553467, 1.5619937181472778, -0.4492126703262329, -0.5595881342887878, 0.4376392364501953, 0.8935688734054565, -1.2548922300338745, 1.0393354892730713, -1.3968446254730225, -0.0874176099896431, 0.990005612373352, 0.719315230846405, 0.14731234312057495, -0.359504759311676, -0.6656674146652222, 0.5231949090957642, 1.5345743894577026, 0.899158775806427, 0.13790783286094666, -2.17104172706604, -0.33096200227737427, -0.7750998735427856, -0.3420770764350891, -1.2503875494003296, 0.638889729976654, 0.8531071543693542, 0.5420663952827454, -0.025196081027388573, -1.4932363033294678, 0.011600309982895851, -1.114877462387085, 0.5490251779556274, -1.6588350534439087, -0.9563581347465515, -0.04664143919944763, -1.8682771921157837, -0.8925962448120117, -1.1338777542114258, -1.1307028532028198, -0.6218907833099365, -0.9806167483329773, -0.6696557402610779, 1.3697699308395386, 0.0006567626842297614, 0.5720267295837402, -0.0754452794790268, 0.6369655132293701, 0.05413859337568283, 0.34711843729019165, 0.15366293489933014, 0.38664522767066956, 1.0123111009597778, -0.3644869029521942, 0.496340274810791, -2.639549493789673, 1.1209617853164673, -0.6926062107086182, 1.0077226161956787, 0.2006937712430954, 0.001301923068240285, 0.31444862484931946, 0.575615406036377, -0.006308675743639469, -1.7151310443878174, 0.551936686038971, -0.5954253077507019, -0.0619257390499115, -0.6535215973854065, 0.9735942482948303, 1.7361122369766235, -1.1823539733886719, 1.596859335899353, -0.11947426199913025, -2.1167004108428955, -0.5731852054595947, 0.2558418810367584, 0.8643666505813599, 0.20975449681282043, 0.09581784904003143, -1.4616034030914307, -1.024487853050232, -0.5602015256881714, 0.32682177424430847, 0.614282488822937, 1.6624993085861206, 2.0511558055877686, -1.5656746625900269, -0.28151601552963257, -0.031381115317344666, 0.7116585373878479, -0.2756335139274597, -0.4714023768901825, 0.8728153705596924, 0.037790216505527496, -0.2716974914073944, 0.7230270504951477, -1.5418823957443237, -2.289083957672119, 1.3261642456054688, 0.0018561271717771888, -0.15534107387065887, -0.176908478140831, -0.10370000451803207, 0.954544186592102, -1.003710150718689, -0.37917447090148926, -0.1699538230895996, -0.5472185015678406, 1.284302830696106, -0.3070657253265381, 1.4477263689041138, -0.4902270436286926, -0.0020832999143749475, -0.4963202178478241, -0.3123086988925934, -0.48012810945510864, 0.5617985129356384, -1.247531533241272, 1.3870805501937866, 0.085124671459198, 0.3593544363975525, 0.3110528588294983, 1.616334080696106, 0.30778053402900696, 1.610935091972351, -0.709060788154602, 0.20622216165065765, -0.741034746170044, 1.2553738355636597, 0.42861130833625793, -0.6913914084434509, 1.349157452583313, -0.28383874893188477, -0.3807459771633148, 1.1411241292953491, 0.4978296160697937, -0.7274386882781982, 0.7718677520751953, -0.33364370465278625, -0.340921550989151, -1.1156266927719116, -0.6997093558311462, -0.627470076084137, 1.4958919286727905, 2.412245273590088, 1.1180833578109741, 2.0018718242645264, 0.451820433139801, -0.398979514837265, 0.849624752998352, -0.5518335103988647, -1.5854308605194092, -0.8180803656578064, -0.7486743927001953, -1.0533369779586792, -1.2314136028289795, -0.8088639378547668, -0.16812214255332947, -0.6592469811439514, -2.115029811859131, -0.0638221874833107, 1.246482014656067, -0.5862312316894531, 0.2662387192249298, 0.3046157956123352, 1.096398949623108, 0.26057666540145874, 0.38030505180358887, 0.37722593545913696, -0.046236831694841385, 1.7859833240509033, 0.21948331594467163, 0.9976152181625366, -1.0300949811935425, 0.33202064037323, 1.1755732297897339, -1.674797534942627, 0.16595415771007538, -1.4494922161102295, 1.6668542623519897, -1.1936534643173218, -0.5545399188995361, -1.2138252258300781, 0.6304528713226318, -0.30085837841033936, 1.721353530883789, -0.9188829660415649, 0.4810146689414978, 0.3617858588695526, -1.2029837369918823, 0.6596531867980957, -0.5789618492126465, -0.453328013420105], [1.2557865381240845, -2.0684947967529297, -0.4063567817211151, -0.8670114278793335, -0.7430413365364075, 1.9978408813476562, 0.13466081023216248, 1.5571619272232056, -0.11943841725587845, 1.0675911903381348, -0.003334884298965335, -0.4949605166912079, 0.03303009644150734, -0.1225757747888565, 1.0475609302520752, 0.4303424060344696, -0.3567703366279602, 0.8353649973869324, -0.9609918594360352, 0.6745526194572449, 0.28252556920051575, -0.6163062453269958, 0.07712072879076004, -1.0308079719543457, -0.9991984963417053, 0.9091637134552002, 1.5576492547988892, 0.1658712774515152, 0.10929311811923981, 0.23334389925003052, 0.5359874963760376, 0.22488103806972504, 0.5237581729888916, -0.07167163491249084, 1.9021055698394775, 1.577431559562683, -0.5449854731559753, 1.6132258176803589, -0.06278114765882492, -0.7025031447410583, 0.028512703254818916, -2.364222288131714, -0.5426009297370911, 0.40766921639442444, 0.5158219337463379, 0.7351596355438232, 0.4948512017726898, -0.7515066266059875, -0.013171156868338585, -0.8639628887176514, 0.9881436228752136, 0.060937654227018356, -0.5379859209060669, 1.0411652326583862, 1.6922552585601807, 0.5102405548095703, 0.005794970318675041, -0.2889944016933441, -0.2287382185459137, 0.7450135350227356, -1.3382229804992676, 1.1019902229309082, -0.5001310110092163, 0.46772199869155884, 0.717076301574707, -0.16127629578113556, -2.0155587196350098, -0.9192443490028381, -0.08153601735830307, 0.36896345019340515, 0.5868852138519287, 1.2695757150650024, -0.8410958647727966, 0.02976350113749504, -0.8414050340652466, 0.5195241570472717, -1.0085551738739014, -0.6402282118797302, -1.7848613262176514, 0.22195972502231598, -1.2452714443206787, 0.6335933208465576, 1.511487364768982, -0.23026113212108612, 0.4855600893497467, 0.12536686658859253, -0.4441665709018707, -0.6960859894752502, 1.1422414779663086, 1.2337433099746704, 0.2724374234676361, -0.02024538815021515, 0.37847796082496643, -1.8336440324783325, 0.057619936764240265, -0.5002861618995667, 0.40623968839645386, 0.16870157420635223, 0.8998387455940247, 1.1424850225448608, 1.2401241064071655, -0.17339058220386505, 0.8972309231758118, -0.2220676690340042, 1.5002119541168213, 0.46535226702690125, -1.5187549591064453, 0.7256155610084534, 0.38036638498306274, 0.2736310064792633, 0.5334938168525696, 0.40803027153015137, -0.6082302927970886, 1.27400803565979, 0.4197220802307129, 0.6026128530502319, 1.38508939743042, -0.27319660782814026, 2.7641801834106445, 0.5164611339569092, 1.4434311389923096, -0.6017560958862305, -0.9565945863723755, 1.6910529136657715, -0.6105426549911499, -0.7653133869171143, -0.23936890065670013, -0.43385449051856995, -0.8871608972549438, 0.8898565769195557, 0.541497528553009, -2.0801258087158203, 0.3623843193054199, 0.22444584965705872, 0.8923075199127197, -2.0203328132629395, -0.015273090451955795, 1.8866690397262573, -0.8461609482765198, -0.1462898701429367, -0.005268976557999849, -0.9984374642372131, 1.9433938264846802, 0.5055516958236694, -1.3307079076766968, -1.7823235988616943, 0.9578163623809814, -0.46289315819740295, 1.9771687984466553, -0.9821733832359314, -0.16837729513645172, 0.8717478513717651, 0.065913125872612, -0.026771169155836105, -0.4390013813972473, 0.2564776539802551, -0.8345035910606384, -0.19766148924827576, -0.3013373911380768, -0.5116323232650757, -1.3466185331344604, -0.8304310441017151, -0.5304198265075684, -1.3508610725402832, -0.9299156069755554, -1.7734630107879639, 0.4508710205554962, 1.8030824661254883, 0.15822669863700867, -0.6185914278030396, -1.040828824043274, 0.47998809814453125, -0.2799516022205353, 1.9027049541473389, 0.5636648535728455, -0.2618825435638428, 0.666126549243927, -0.40979301929473877, 0.04496517777442932, 0.7675893306732178, 0.4794723093509674, 0.5909572839736938, 0.4373305141925812, -1.0750329494476318, -0.9014925360679626, -0.8813417553901672, 0.5631608366966248, -1.5617001056671143, -2.050266742706299, 2.2182776927948, -0.013226144015789032, 0.22237862646579742, 1.0123226642608643, -0.15672731399536133, 0.9208284616470337, -1.824402928352356, 0.48209595680236816, -0.8765843510627747, 1.1028081178665161, 0.8627221584320068, 0.08824092149734497, -0.037557344883680344, 0.6918830871582031, -0.1231985092163086, -0.3928452432155609, -0.3878979980945587, -0.09483630955219269, 0.3675018548965454, -2.655693769454956, 1.0671687126159668, 0.7102853655815125, -1.0225075483322144, -2.9975969791412354, -0.5737111568450928, -1.0904656648635864, 0.4392137825489044, 1.1449356079101562, -2.5416510105133057, -0.5050147175788879, 1.10712730884552, -0.9778109788894653, -0.11234341561794281, 0.5449824333190918, 1.136260986328125, 0.6883846521377563, 0.38984057307243347, 0.7383680939674377, 0.6452279090881348, 0.8595350980758667, -0.7356711030006409, 2.6464080810546875, -0.6400731801986694, -0.27961495518684387, 0.36770740151405334, 1.7737380266189575, 1.2196800708770752, -1.7342478036880493, 1.05294668674469, 1.5081968307495117, 1.1743043661117554, 0.6159741878509521, -0.33907556533813477, -0.11515920609235764, -1.9911295175552368, -1.1970829963684082, -1.6464006900787354, 1.1302434206008911, -1.4908989667892456, 1.2480309009552002, 0.12897641956806183, 0.7659231424331665, -1.1577446460723877, -0.14011944830417633, -0.8649643063545227, 1.8447482585906982, -0.2131296992301941, -0.3361298143863678, 0.18040607869625092, 0.9998043179512024, 0.6780079007148743, -1.0434147119522095, 0.5650557279586792, -0.757046103477478, -2.297912359237671, 1.6956219673156738, -1.12274968624115, 0.5369067788124084, 0.9469383358955383, -1.3880040645599365, 0.6869599223136902, -0.3231104016304016, -1.210791826248169, -1.9765071868896484, -0.42177996039390564, 0.4083157181739807, 1.5603855848312378, 1.251213550567627, -0.9156559109687805, 0.989348292350769, -0.6336744427680969, -0.27195075154304504, 1.1249655485153198, -0.7919777631759644, 0.3989650309085846, 0.07218436151742935, -0.005162160377949476, -0.07279646396636963, 0.26036199927330017, -1.333296298980713, 1.5918595790863037, 1.9268971681594849, 0.47782042622566223, -1.4288634061813354, -1.8582853078842163, 1.196876883506775, -1.1756192445755005, -0.5351482033729553, 0.6026254892349243, -1.9945873022079468, 0.21736641228199005, -1.14407479763031, 0.10071588307619095, 0.37278103828430176, 0.1862834095954895, 2.151700973510742, -1.7501798868179321, 0.7151172161102295, 0.12241688370704651, -0.6684063076972961, 0.6323005557060242, 0.579842209815979, 0.7087457776069641, -1.5531163215637207, -0.6036704182624817, -0.2782494127750397, 0.4332549273967743, -0.718749463558197, -0.695830225944519, 0.96414715051651, 0.8966236710548401, -1.6222025156021118, 0.132755845785141, 0.5429229140281677, -0.22663719952106476, 1.8983229398727417, 0.00746960798278451, 0.5434188842773438, -0.6779482960700989, -0.08641091734170914, 0.01008948776870966, 1.2086392641067505, -1.508622169494629, -0.6083692908287048, -0.05508342757821083, -2.684645891189575, -1.5908151865005493, 0.6755158305168152, -0.641093909740448, 0.016683219000697136, 1.005007266998291, -0.3893977403640747, -0.5805266499519348, 1.2980892658233643, 1.1431784629821777, 1.7316585779190063, 1.4502228498458862, 1.9056637287139893, -0.22841082513332367, -0.693996012210846, -0.6402612328529358, -1.3916832208633423, 0.09355344623327255, -0.8837441205978394, 0.3927403390407562, 0.10932860523462296, 0.3170558512210846, -2.139173746109009, -1.8343714475631714, 1.7276400327682495, -0.03214452788233757, 1.1374644041061401, 0.22883227467536926, -0.5430386662483215, -0.038008350878953934, 1.138420581817627, 1.5031297206878662, 0.46834611892700195, -1.9038249254226685, 1.2336348295211792, 1.381739854812622, 0.08786148577928543, 2.0557940006256104, 0.13983555138111115, 0.6762700080871582, 0.6703869700431824, 2.435049057006836, 0.45122030377388, 1.049033284187317, -1.1542426347732544, 1.4718997478485107, 1.2356747388839722, -0.7174345254898071, 1.1938495635986328, -0.19154323637485504, 0.556910514831543, -0.5544491410255432, 0.1853775978088379, -0.686700165271759, 0.4744028151035309, 1.881957769393921, -0.28329965472221375, -0.2014709711074829, -0.9195443987846375, 0.020382588729262352, 0.4715103805065155, -3.1307315826416016, -3.535997152328491, 0.9826725721359253, 0.6398016810417175, 1.0145257711410522, 1.0857490301132202, -0.7977268695831299, 0.6059161424636841, -0.069132499396801, 0.43570801615715027, 0.137899711728096, -1.3888041973114014, 2.3848259449005127, -0.5006392598152161, 1.3462988138198853, 1.2778416872024536, -0.27100875973701477, 0.1488444209098816, 0.5089408755302429, -0.06529298424720764, 1.5065360069274902, 0.4666047990322113, 0.11927317827939987, -0.9445071220397949, -0.36464256048202515, -0.3163221776485443, 1.1733931303024292, -0.6284651756286621, -0.9216234683990479, 0.43613559007644653, -1.4453434944152832, -1.6122336387634277, -0.48463791608810425, -1.04159677028656, 0.991575300693512, -0.0031446388456970453, 0.16557452082633972, 0.1281120330095291, -0.7087015509605408, -0.3034195601940155, 0.4347846210002899, 0.3930628001689911, 1.3039993047714233, 3.12546968460083, 1.0299534797668457, -0.5504591464996338, 1.4417561292648315, -1.3067078590393066, -1.8507758378982544, 1.309446930885315, 0.12388955801725388, 0.9514092803001404, -0.5568487048149109, -0.3156220316886902, 0.10714101791381836, 0.8053101897239685, 0.19277729094028473, 0.38694679737091064, 1.3248289823532104, -3.4728987216949463, 0.8137620687484741, 0.7918543219566345, 1.143965482711792, -0.7886031270027161, -0.25080639123916626, 0.4140544831752777, -1.3776140213012695, 0.5461294651031494, -1.4701303243637085, -0.09562316536903381, -0.03323480486869812, -0.7505227327346802, -0.03392019122838974, 0.2574813663959503, 0.3810563087463379, 0.44126805663108826, -1.266440987586975, 0.5667892098426819, 1.6838266849517822, -0.29471367597579956, -0.3679271936416626, -2.044482469558716, 0.021669160574674606, -0.18797722458839417, -1.7086037397384644, 2.452099561691284, -0.30457788705825806, -1.0093857049942017, -0.5478082299232483, 0.2850547432899475, 0.30382367968559265, -1.0536234378814697, -0.9718211889266968, -0.10637865960597992, 1.2351080179214478, -0.982184112071991, 0.49896007776260376, -0.4652036130428314, -1.4614769220352173, 0.01954135112464428, 0.5722175240516663, -0.14698927104473114, 0.273615300655365, 0.852250874042511, 0.772045373916626, -1.7885686159133911, 1.9572502374649048, 1.122564673423767, 0.06136230006814003, -0.2716258466243744, -0.34058743715286255, -0.9655360579490662, 0.3536967933177948, -0.8640178442001343, 0.5485227704048157, -0.2966957688331604, -0.19331978261470795, -0.34749871492385864, -0.14365887641906738, 0.2582516074180603, 0.2159745991230011, -0.315726101398468, -1.1064536571502686, -0.022151095792651176, -0.6543645858764648, -0.596793532371521, -1.8505159616470337, 0.24285900592803955, -0.24886639416217804, 2.3644981384277344, -0.02051647938787937, -0.3137056231498718, 0.17036248743534088, 0.4311691224575043, 0.5072920918464661, 0.3452158570289612, -0.07345253974199295, -1.7380064725875854, 0.3472331166267395, 0.9783287048339844, -1.8633664846420288, -0.3263776898384094, 0.406737357378006, -0.7281154990196228, 0.326198548078537, 0.0967516154050827, 0.1244036927819252, 0.49678850173950195, 0.5831171870231628, 1.199276328086853, 0.42628738284111023, -1.0480389595031738, -0.7983296513557434, 0.7765178680419922, -0.9558270573616028, -0.3260038197040558, -1.5394625663757324, 0.6117433309555054, 1.159517526626587, 0.9699972867965698, -0.16853067278862, -0.9966393113136292, -0.9971486330032349, -0.4052295386791229, -1.0975873470306396, 1.337733268737793, -1.641809344291687, 0.6892521381378174, 1.3523072004318237, 0.08806072920560837, -0.6382959485054016, 1.7113803625106812, 1.2362840175628662, 0.4859909117221832, 0.09547744691371918, -1.5001407861709595, 0.6822060346603394, 0.2954272925853729, 1.071669340133667, 2.219695806503296, -0.5320274829864502, 0.045637089759111404, -0.5714623928070068, -0.5160805583000183, -0.4446018934249878, 1.637669563293457, -0.019860701635479927, 0.3232218623161316, 0.4033832550048828, 1.2113717794418335, 2.0815441608428955, 0.42883428931236267, 0.2298799604177475, -1.0081501007080078, 0.19541940093040466, -0.2650218605995178, -0.9533370137214661, -0.22700105607509613, 0.6119566559791565, -0.5931723117828369, -0.676583468914032, -1.0457229614257812, 0.8643524050712585, -0.04616398736834526, 1.1138790845870972, 0.49975714087486267, -1.1960362195968628, 0.8361635804176331, 0.7680382132530212, 0.4876335859298706, 0.48364970088005066, 1.350089192390442, -1.6614748239517212, 1.6601790189743042, -0.06137509271502495, -0.09523016214370728, -0.8021228909492493, 0.43126776814460754, 0.11082621663808823, 0.8031242489814758, -0.7367947697639465, 0.04830159619450569, -1.2361301183700562, 0.6110685467720032, -0.05676575005054474, -1.3346631526947021, -0.6070425510406494, -1.9838148355484009, 0.3377591669559479, -0.6810932755470276, -1.0633000135421753, 0.03176615759730339, -0.18387572467327118, -0.3742659389972687, 0.9622014164924622, 0.7794015407562256, -0.7401731610298157, 0.6181732416152954, -0.7974170446395874, -0.36025965213775635, 0.8768477439880371, -0.036343760788440704, -1.7980694770812988, -1.2994827032089233, 1.0653138160705566, -1.0569822788238525, -1.0041511058807373, -0.654248058795929, 0.22521638870239258, 1.359208345413208, -0.2531997263431549, -0.13886408507823944, 1.6812254190444946, 0.6109277606010437, 0.7962004542350769, -0.38258010149002075, 0.42433327436447144, 1.0467103719711304, -0.9090477228164673, 0.41646450757980347, 0.5378230810165405, -1.0669398307800293, -0.2656654417514801, -2.2130320072174072, -0.26619255542755127, -0.6937111616134644, 0.15283377468585968, -0.7441413998603821, 0.2931620180606842, 0.7203972935676575, -1.0436204671859741, 1.2315542697906494, 0.16030003130435944, -1.629754662513733, -1.89328134059906, 1.2211295366287231, 0.8886183500289917, -1.2747102975845337, -0.5977578163146973, -0.1769103854894638, 0.13119378685951233, 0.3035564720630646, -0.14425873756408691, -1.3591020107269287, 0.07002600282430649, 1.2569658756256104, 0.2360592931509018, -0.41747546195983887, -1.657913088798523, -1.879417896270752, 1.1729216575622559, 0.7760778069496155, -0.03646212816238403, 0.5089807510375977, -0.4612136483192444, 0.2216642200946808, 0.39785775542259216, 0.6024443507194519, -0.025120068341493607, 0.22583049535751343, -1.2533165216445923, 1.068340539932251, 0.17495276033878326, -0.4694584310054779, -0.2976336181163788, -1.076562762260437, -0.9002344012260437, 0.1966252475976944, -0.30446916818618774, -1.255886435508728, -0.0385965071618557, 0.24474172294139862, 1.1123785972595215, 1.2178758382797241, 0.4784732162952423, -0.9229980707168579, -0.5950024127960205, 1.0099178552627563, -0.28521594405174255, 1.0705000162124634, 0.11016136407852173, 0.4470081627368927, -0.24522042274475098, 0.7389283180236816, 1.6802024841308594, 0.16997414827346802, 0.7448983788490295, 0.02859891764819622, 0.7500612735748291, -0.07710778713226318, -1.278709888458252, -0.6905866861343384, -1.1579772233963013, 1.0849888324737549, -0.601551353931427, 0.482191801071167, -0.4878222942352295, 0.48863929510116577, -0.12876944243907928, -0.2181539535522461, -0.053399067372083664, 1.142993688583374, 1.8885523080825806, -0.8920565247535706, 2.4930245876312256, -1.721558928489685, 1.4480921030044556, -0.4136255979537964, -0.11604966223239899, 1.9178473949432373, 1.1566708087921143, 1.4527078866958618, -0.5966179966926575, 1.5160510540008545, 0.667095422744751, -0.04135163128376007, 0.5516773462295532, 1.7198914289474487, 0.6714573502540588, -0.5073096752166748, 0.8136584758758545, 0.2943529188632965, 1.5317572355270386, -1.4252874851226807, -0.41459232568740845, 0.19062435626983643, 1.4283016920089722, -0.18802934885025024, -0.3418844938278198, 0.46102872490882874, 0.1716991662979126, -0.15866367518901825, -2.1985576152801514, -1.301255464553833, -0.8208286762237549, 0.4956691563129425, -1.387269139289856, -1.0944201946258545, -1.0206663608551025, -0.8047949075698853, 0.851423442363739, -0.9584177732467651, 1.5040223598480225, 0.2540123462677002, 0.23675625026226044, 0.7760681509971619, -0.053369734436273575, -0.44377270340919495, -0.3522243797779083, -0.7570337057113647, 0.7549434900283813, -0.27780681848526, -0.3042716383934021, 0.8731003999710083, 0.4669283628463745, -0.6656144857406616, -0.5562455058097839, 0.4841477870941162, -1.070703148841858, -0.2499718964099884, -0.6117830872535706, 0.5737754106521606, 0.2320868968963623, 1.4163130521774292, -1.0621776580810547, 1.1256859302520752, 0.9238421320915222, 1.2728358507156372, 0.24197673797607422, 0.34631383419036865, 0.018302418291568756, -0.521221399307251, -1.7470452785491943, 0.4460386633872986, 0.012058778665959835, -1.5229811668395996, 0.49770110845565796, 0.8090693950653076, -0.4703710079193115, 0.26574867963790894, 0.7206186652183533, -1.2514005899429321, 1.1687201261520386, -0.16821561753749847, 0.3013225495815277, 1.2061665058135986, 1.554646372795105, -1.002315640449524, -1.4242932796478271, 0.8238529562950134, 1.7577893733978271, 0.9820052981376648, 1.7836536169052124, 0.07813935726881027, -0.306179404258728, 0.37866005301475525, -0.6111698746681213, -2.0095109939575195, 0.5963079333305359, -0.8130584359169006, 0.547644317150116, 1.338595986366272, -0.7820740342140198, -0.12813666462898254, 0.35031628608703613, 0.9833059310913086, -0.0571955144405365, 0.3630930483341217, -0.7659985423088074, 0.09567689895629883, -0.36597099900245667, 2.320608377456665, -1.4075580835342407, 1.706986427307129, 1.2083951234817505, -1.0553752183914185, 0.982406735420227, -0.5420653820037842, 0.087470643222332, -0.9160977602005005, -1.3296372890472412, -0.3426052927970886, -0.03237203136086464, -0.43729251623153687, 1.385374903678894, -0.4145687222480774, 0.24854150414466858, -0.45249783992767334, 0.507878839969635, -0.9591660499572754, 0.014384065754711628, 0.09429237246513367, -1.3728039264678955, 1.3824388980865479, 0.9304600358009338, 0.9533178210258484, -0.31205686926841736, -0.251864492893219, -0.5343970656394958, -0.00982209388166666, 0.13931630551815033, -1.4151782989501953, 0.8678082227706909, -1.5177550315856934, 1.0164886713027954, -0.3101416528224945, -0.7914867997169495, -1.4950909614562988, -0.7145296335220337, 2.0710792541503906, -1.2294268608093262, 1.5645726919174194, 0.1646188199520111, 0.25534841418266296, 2.7019217014312744, -0.38979408144950867, -0.22970375418663025, 1.048580527305603, 1.4426755905151367, 0.3609241247177124, 1.4027386903762817, 0.4092666506767273, 0.7559411525726318, 0.6269489526748657, -0.5315399169921875, 0.531470537185669, 0.3776339292526245, 3.0185210704803467, -1.1747140884399414, -1.1646206378936768, 0.5239425897598267, -0.13896194100379944, 0.003408749122172594, -0.6489220261573792, -1.1394047737121582, -1.017946720123291, -1.746313214302063, -0.8229546546936035, -1.7055691480636597, -0.16159839928150177, -0.1690247505903244, -0.010742777958512306, -0.2664155066013336, 1.3057384490966797, -1.4582549333572388, 0.3077773153781891, -0.5120163559913635, -1.1801936626434326, -0.7664560675621033, -0.9610057473182678, 0.5090962052345276, 0.8976677656173706, 0.2248297482728958, 1.0478743314743042, -1.2035623788833618, 1.0204728841781616, 0.5111463069915771, -0.6411082744598389, 0.4027195870876312, -0.6355827450752258, 0.3193187415599823, -2.0914011001586914, -0.9971835613250732, 0.8758289813995361, 0.06268420070409775, -0.11795204877853394, 0.6672664284706116, -0.10589264333248138, 0.9748737812042236, -0.8362692594528198, -1.2851569652557373, 1.131103754043579, 0.683159351348877, -1.1753119230270386, 0.7326815128326416, -0.28995123505592346, -0.402786523103714, 0.9107019305229187, 1.7038251161575317, 2.1195733547210693, 0.03312727063894272, 0.031793832778930664, 0.012889839708805084, -1.4270317554473877, -2.45558500289917, -1.132718563079834, -0.4088979661464691, 1.962224006652832, 0.15287695825099945, 1.142196774482727, -1.7024215459823608, -1.3205293416976929, 0.41251063346862793, 1.1874910593032837, 2.3735530376434326, 2.4952003955841064, 0.7289320230484009, -0.1609322726726532, 0.020037280395627022, -1.0089067220687866, -0.9646157026290894, -0.8342829942703247, 1.2894785404205322, -0.8826370239257812, 1.5389652252197266, -1.1056203842163086, -1.404664397239685, 0.34088191390037537, -0.22411593794822693, 0.04735575616359711, 1.0049340724945068, -1.69637930393219, 0.13867931067943573, -2.3199713230133057, 0.6680832505226135, 0.3129151165485382, 0.30402418971061707, 0.342923104763031, -0.9422465562820435, 1.7666314840316772, 1.7229121923446655, 0.2677719295024872, -0.5674059987068176, 2.1999564170837402, 0.2872239053249359, -0.9238824844360352, 0.5136193633079529, -1.7880159616470337, 2.1092703342437744, 0.47943568229675293, 0.7511904239654541, -0.5985000729560852, -0.14379848539829254, 0.20031993091106415, 0.7676536440849304, 1.3819600343704224, 0.6341972351074219, -1.6678804159164429, -0.4317149221897125, -0.5101004838943481, -0.12450245767831802, 0.4504778981208801, 0.8631067276000977, 0.8347351551055908, 0.2989511787891388, -0.3241083323955536, -0.28026825189590454, 0.5288412570953369, 1.00810968875885], [-0.8915706276893616, 2.38444447517395, 1.5918502807617188, -1.5123891830444336, 0.9777836799621582, -1.4620997905731201, -0.609627366065979, -1.3288352489471436, -0.054590854793787, 1.030856966972351, -1.9630974531173706, -0.3178131878376007, -1.056583285331726, 1.3310272693634033, 0.038565367460250854, -0.18697115778923035, 0.019610486924648285, -0.6187267899513245, -0.527543842792511, -0.4350550174713135, 2.1197447776794434, 0.4230072498321533, -0.8851765990257263, -0.05165940150618553, -0.21551074087619781, -0.1766389012336731, 1.5478177070617676, -0.4879828691482544, -0.589513897895813, -1.0054352283477783, -0.6354424357414246, 1.3888018131256104, -1.2799290418624878, 0.19712872803211212, -1.1246706247329712, -2.294379949569702, -0.5334913730621338, -0.48225080966949463, -0.11098963022232056, 0.5217331647872925, -0.962554931640625, 0.34710678458213806, 2.2977185249328613, 0.21323652565479279, -0.026490459218621254, 0.5837901830673218, 0.5987586379051208, -0.27334651350975037, 0.16047993302345276, 1.704056978225708, 1.1002247333526611, -0.8741042613983154, -1.1045866012573242, -1.202518343925476, 1.7788327932357788, -0.574298620223999, 2.4600117206573486, 1.2643494606018066, 0.3162235915660858, -0.12794354557991028, 0.6554149389266968, 0.7177921533584595, 0.00862694438546896, 0.2783087193965912, 0.16678714752197266, 0.507564127445221, 0.5014721751213074, -0.16627217829227448, 0.36363494396209717, 0.7495606541633606, -1.11415433883667, -0.29585206508636475, -0.6663097739219666, 0.747774600982666, -0.5845764875411987, 0.9359567761421204, 1.8591655492782593, -0.6543508172035217, 0.43010586500167847, -1.2846217155456543, -0.04658699035644531, -2.3162026405334473, 1.7721660137176514, 0.14097775518894196, 1.7758753299713135, -1.16636061668396, 0.9450899362564087, -1.2082585096359253, -0.09096017479896545, -1.0164408683776855, -0.6532536745071411, -0.5843726396560669, -0.4756016433238983, 0.9910743832588196, -0.750238835811615, -0.17595356702804565, 1.4930565357208252, 0.7245230674743652, -0.02235540747642517, 0.4358823001384735, 0.3002556562423706, 0.03730182349681854, 1.2393385171890259, -1.669826626777649, 0.5654671788215637, 2.05501127243042, 0.2685229778289795, 0.26712867617607117, -1.3490208387374878, -0.35571521520614624, -0.3282366096973419, -0.4618050456047058, 2.1307876110076904, 0.3781696856021881, 0.9171886444091797, -0.10810777544975281, -0.8247740268707275, -0.5072236657142639, 0.17669890820980072, 0.8913973569869995, 0.9700864553451538, 0.8054279088973999, -1.4305013418197632, -0.6647630333900452, -0.496008038520813, 0.5334978103637695, 1.0329539775848389, 0.2840224802494049, -0.5638996362686157, -0.512509822845459, 1.183354377746582, 1.4885436296463013, -1.529727816581726, 1.069700002670288, 2.1848857402801514, -0.3424641191959381, 0.6274988651275635, -2.006957530975342, -1.0288666486740112, -0.05473695322871208, -1.9355651140213013, -0.6222900152206421, -0.5135005116462708, 0.7161325216293335, -0.0896090418100357, 3.0050466060638428, -0.35152488946914673, -0.5159348249435425, 0.04822811856865883, -0.24715811014175415, 0.4280485510826111, 1.3948290348052979, 2.8386311531066895, -0.6927662491798401, -1.0974191427230835, -0.4044806957244873, 0.9420191645622253, 0.8529724478721619, 0.8903481364250183, -1.895413875579834, 1.5416748523712158, 1.3658851385116577, 0.9077418446540833, 0.836697518825531, 0.12276425212621689, 0.8340135216712952, -0.9470635652542114, 0.6157175898551941, 0.044698648154735565, -0.2773721516132355, 0.563624382019043, 1.2565932273864746, -0.047220487147569656, 0.7664767503738403, -0.8221083879470825, 1.5663213729858398, 0.7142999172210693, -0.8915939927101135, 0.6166954636573792, 0.050549305975437164, -1.6539956331253052, 0.9161714911460876, -1.8757500648498535, 1.1449445486068726, 0.9874671697616577, -0.8808014392852783, -0.7849802374839783, -0.44234174489974976, -0.41012272238731384, 0.8082088828086853, -0.09317472577095032, -1.5223115682601929, -1.9763296842575073, -0.15300068259239197, -1.2961573600769043, 1.2373931407928467, -0.11780261248350143, -0.6054378747940063, 0.16347216069698334, 0.01877281628549099, 0.01727679744362831, 1.6737240552902222, 1.6123243570327759, 0.18725347518920898, 0.8322516083717346, -0.440761923789978, 1.8311395645141602, 0.4390818178653717, 0.8270795941352844, -0.2591421902179718, -0.7925481796264648, -0.7231085896492004, 1.5435171127319336, 0.026411699131131172, -0.48995283246040344, 0.9968022704124451, -0.3691185712814331, -2.3338000774383545, -0.6663486957550049, 0.0927106961607933, -0.7117033004760742, -1.4903600215911865, 1.4589262008666992, -0.5259044170379639, -0.7785415053367615, 0.505384087562561, 2.6650173664093018, -0.825577437877655, -0.07528228312730789, -0.8699819445610046, -0.17120327055454254, -0.24336303770542145, -0.4050839841365814, -0.13366295397281647, -0.6721583008766174, -0.46559590101242065, -0.2339765429496765, -1.0357325077056885, 0.5732767581939697, 0.6255543828010559, 0.011614756658673286, -1.162348747253418, -0.838639497756958, 1.8346272706985474, -1.4812166690826416, -0.17077918350696564, -0.7579079270362854, 0.07236842066049576, -0.1603696346282959, 0.0029635136015713215, -0.9129960536956787, -0.0854855626821518, -0.015095614828169346, 0.29749006032943726, -1.1296435594558716, -1.43171226978302, 1.4709261655807495, 2.372986316680908, 0.0862475112080574, 0.5164704918861389, 1.1971282958984375, -0.6684178709983826, -0.4818821847438812, -0.7145087718963623, 0.40639734268188477, 0.7739386558532715, -0.2626642882823944, -1.872600793838501, 0.24129213392734528, -1.8259844779968262, -1.0874987840652466, 0.8100533485412598, -1.6101186275482178, -0.6647700071334839, -0.0039618187583982944, -0.382529616355896, 0.48152220249176025, 0.6012044548988342, -0.04201285168528557, -1.5362062454223633, -0.5129610300064087, 1.3533108234405518, -1.9091227054595947, -0.3451457619667053, -0.2230292558670044, -0.6676712036132812, 1.22201406955719, 0.14483694732189178, -0.7900465726852417, 0.05512687563896179, -0.652333676815033, -1.1400368213653564, -2.762274742126465, -0.14239457249641418, -2.3056275844573975, 1.082006573677063, 0.2929094731807709, -0.6491611003875732, 0.2399817854166031, -0.8516499996185303, -1.855143666267395, -0.7930868268013, 0.07723737508058548, -0.4467010200023651, -0.7877309322357178, 0.3766171634197235, 0.192471444606781, 2.095250129699707, 0.16327004134655, 0.6584251523017883, 1.2769986391067505, -1.0712199211120605, -2.7211029529571533, -0.04405003413558006, 0.21750859916210175, -0.8275578618049622, 0.45012030005455017, -0.7230467200279236, -0.9422889947891235, 0.34777483344078064, 0.28854718804359436, 1.3706855773925781, -0.14690715074539185, 0.2669689655303955, 1.1553301811218262, -1.884431004524231, -0.843499481678009, 0.18482910096645355, 0.6798586845397949, -0.39261823892593384, -1.4370276927947998, -0.1657150685787201, -0.4803120195865631, 2.7260050773620605, -1.0417348146438599, 1.3570507764816284, 0.28244084119796753, 0.7269367575645447, -1.1626484394073486, 0.9303867220878601, 1.710653305053711, -0.05436711385846138, -0.22997984290122986, -0.5042270421981812, 1.1858625411987305, -0.7980771064758301, -0.3510010540485382, -0.6632342338562012, 1.2502808570861816, -0.43689462542533875, 0.2043699473142624, -0.919938862323761, 1.0596462488174438, 1.4593178033828735, -0.9505046010017395, 1.7185262441635132, -0.880190372467041, 1.2866754531860352, 0.20487625896930695, -0.976793646812439, 0.5787031054496765, 1.7698241472244263, 2.0283803939819336, -0.07305701076984406, 2.745852470397949, -0.45810768008232117, -0.9055707454681396, -1.4984530210494995, 0.6943126916885376, -0.12914329767227173, -0.510977566242218, -0.7682316899299622, -0.15808984637260437, 0.17571361362934113, -0.6134492754936218, -0.36373499035835266, -0.8402522206306458, 0.613578200340271, 1.276650309562683, 0.5897032618522644, 2.0549204349517822, 1.5996136665344238, 0.4074794352054596, 0.49816280603408813, -0.250978946685791, -0.33831119537353516, 0.13167643547058105, 0.30829790234565735, -1.762740135192871, -0.9905954599380493, 0.05401664972305298, 0.1264290064573288, -0.7839584946632385, -0.3447623550891876, -1.3463990688323975, -1.101497769355774, 1.121254801750183, -0.27324286103248596, -0.7189725637435913, -2.2670795917510986, 1.1651774644851685, 0.05498418211936951, -1.370278000831604, -0.9236600995063782, -0.29992708563804626, -1.6640150547027588, -0.5463575720787048, 0.3915659785270691, -0.47275668382644653, -0.6666756868362427, 0.8385119438171387, -0.9996481537818909, -0.25518736243247986, -0.22266584634780884, -1.7939027547836304, 0.42952796816825867, 0.7786117196083069, 0.5301119089126587, 1.7264294624328613, 0.6530079245567322, -0.33153048157691956, -0.9214776158332825, -1.11976158618927, -0.4419912099838257, -0.8202601075172424, -0.3092309832572937, -1.0182267427444458, -0.524784505367279, 1.2392642498016357, -0.6536718010902405, -1.0114209651947021, -1.581337809562683, 0.17202259600162506, -1.1529207229614258, -1.9980305433273315, -0.3334912955760956, 0.4279312193393707, -0.4188549816608429, 1.0870559215545654, -0.14470769464969635, -0.3245645761489868, -0.13150067627429962, 0.3205684721469879, 0.8868856430053711, 1.6045621633529663, -1.8807628154754639, 0.1008969247341156, 0.38349950313568115, 2.2830944061279297, -1.2702652215957642, -0.9412210583686829, -0.09935226291418076, 1.1183525323867798, -0.11081250011920929, -0.21713033318519592, -0.06790497899055481, 0.6670410633087158, -0.6588680148124695, 0.34131574630737305, -0.17477795481681824, -0.03380950540304184, 0.6835082769393921, -0.697589635848999, -0.3250317871570587, 0.630951464176178, -0.2082584947347641, 0.4225488007068634, 0.23491355776786804, 0.01585436798632145, -1.0497385263442993, -0.01713835820555687, 1.4155575037002563, -2.3257369995117188, -1.4264734983444214, -1.0926239490509033, 0.18868175148963928, 0.13319876790046692, -1.540489912033081, 1.4676119089126587, 0.7692971229553223, 0.4707203805446625, 0.6440911889076233, 1.5505315065383911, 1.4921354055404663, -0.504891574382782, -1.085937738418579, -0.12071729451417923, 0.3320627212524414, -0.008536762557923794, -0.48996487259864807, 0.9475441575050354, 2.212665557861328, -0.6181773543357849, -0.1618286669254303, 0.550122082233429, 0.3301282227039337, 0.026069914922118187, -0.5639973282814026, -0.8233925700187683, 0.24298283457756042, -0.16914910078048706, -0.012172577902674675, -0.7734301090240479, 0.0012746925931423903, -0.7137600779533386, 1.5932930707931519, 0.6374655365943909, -0.6076979041099548, -0.1857488453388214, 1.0413562059402466, 2.608119249343872, 1.3922977447509766, 1.656195878982544, -0.8222436308860779, 0.8906682133674622, -1.396826982498169, -1.3006402254104614, 0.140593022108078, -0.5738897919654846, -0.7916121482849121, 0.1463971734046936, -0.10171925276517868, -0.9011199474334717, -0.619356632232666, -0.7379881739616394, 0.1480344533920288, -0.9722493886947632, 2.419752597808838, 0.46088358759880066, -0.6182569265365601, -0.9157840609550476, 0.5002535581588745, -0.02930585853755474, 0.8462346792221069, 2.1329424381256104, -1.15647554397583, 0.2780792713165283, -1.1270289421081543, -2.0706117153167725, -0.6745002865791321, -0.19895006716251373, 0.6757384538650513, -1.6360231637954712, -0.7093942761421204, 2.090984582901001, 0.9600977301597595, 0.7409811615943909, 1.7514970302581787, 0.21222437918186188, 0.520092248916626, -0.23107798397541046, -0.7836256623268127, 0.6243064403533936, -2.425248861312866, -1.132658839225769, 2.478151559829712, -0.019820723682641983, 0.2606140375137329, 1.6906702518463135, -0.5812812447547913, -0.20166708528995514, -1.26865816116333, 1.1429609060287476, 0.6979203820228577, -0.4791559875011444, 0.5892664194107056, 0.21633534133434296, -0.42491140961647034, -0.24550127983093262, 1.0246798992156982, -0.3365060091018677, 1.3565257787704468, 2.1574699878692627, 1.0658975839614868, -0.41392141580581665, -0.3377692401409149, -0.9124420285224915, -1.245442509651184, 0.2390081137418747, -1.7449232339859009, -0.2979442775249481, 1.439263105392456, -0.6662367582321167, -0.2220005989074707, 0.617821216583252, 0.6282175183296204, -0.6649539470672607, -0.4586327075958252, -2.661971092224121, 0.6146142482757568, 0.622861385345459, -0.6555490493774414, 0.966194212436676, -0.7058057188987732, 1.2277356386184692, 1.4630846977233887, -1.3730958700180054, 1.3147941827774048, -0.32664787769317627, -0.05555737018585205, 0.93616783618927, 0.818751335144043, 1.158446192741394, -1.166885256767273, -1.1747968196868896, 0.4456636607646942, 0.7839879393577576, -1.2897980213165283, -0.39685696363449097, -0.4337952733039856, 2.430899143218994, -0.8940240740776062, -0.9597553014755249, -1.1255457401275635, -1.2939412593841553, -0.7445770502090454, -1.1628432273864746, -1.439913272857666, -0.9983378052711487, 1.2927405834197998, -0.17505672574043274, 0.3544713854789734, -1.155619502067566, 0.18358589708805084, 1.3833105564117432, 0.06732897460460663, -1.8655123710632324, -1.5846303701400757, -0.7686607837677002, 0.39464882016181946, 1.2545374631881714, -1.953177809715271, 0.957646906375885, -0.9072934985160828, -1.851514458656311, 0.31618452072143555, 0.3210694491863251, -1.9106653928756714, -0.9500000476837158, -1.7149279117584229, 1.253969430923462, 0.8184308409690857, 1.8575068712234497, 0.6252552270889282, -0.08156712353229523, 0.6005666255950928, 1.1373021602630615, 0.5643402934074402, 0.038199104368686676, 0.015128706581890583, -0.6664703488349915, 0.33462896943092346, -0.3866853713989258, 0.6958537697792053, 0.12018179148435593, 2.321918249130249, 0.4882686138153076, -0.005526693072170019, -0.0974750965833664, -0.16912724077701569, -1.3253872394561768, 0.8288968801498413, -0.6561275720596313, 0.7007342576980591, 0.22106605768203735, 0.5796526074409485, 0.48036643862724304, -0.22246690094470978, 0.4407530128955841, -1.1056201457977295, -1.1914201974868774, 0.9409607648849487, -0.8533217310905457, -0.40785640478134155, 0.6384809613227844, -0.04132074490189552, 1.5042287111282349, -0.08450955897569656, 0.039174262434244156, 1.9468871355056763, -0.35773953795433044, 0.231188103556633, 0.9281019568443298, 1.1153370141983032, -1.1445547342300415, 0.9414113759994507, -1.225657343864441, 0.24601198732852936, -0.7300516366958618, -0.4201337993144989, -0.713579535484314, -0.23088757693767548, -1.400302767753601, -1.2371379137039185, 0.841088593006134, 1.810714840888977, 0.239670991897583, -1.2370144128799438, 1.8640022277832031, -0.36207157373428345, -1.5971484184265137, 1.2753973007202148, -0.4004382789134979, 0.3733018636703491, 0.3616452217102051, -0.24726389348506927, 0.8695194721221924, -0.7787852883338928, 0.832037627696991, 0.6820818185806274, 1.4860368967056274, -0.695285975933075, -0.9967739582061768, -1.5226560831069946, -0.4841311275959015, 0.2951402962207794, 1.2826045751571655, -0.7950426936149597, -0.8429180383682251, 0.9141882658004761, 0.9204348921775818, -1.4308993816375732, 1.010553240776062, -0.694342315196991, -1.1234296560287476, 0.14926448464393616, -1.2000865936279297, -0.9597766399383545, -1.1150842905044556, -1.391094446182251, -2.1651864051818848, -0.7529721260070801, -0.8555113673210144, -1.9591761827468872, 0.407804399728775, 0.05551252141594887, -0.4306361973285675, -0.11092039942741394, 0.3635685443878174, -2.8014464378356934, -0.09198106080293655, -0.14688245952129364, -0.015895646065473557, -1.219464898109436, 1.1375763416290283, -1.3611217737197876, 0.022557521238923073, -0.5593360662460327, -0.9866926670074463, -1.2129080295562744, -0.9830681085586548, -0.5676770210266113, -0.8454394936561584, -1.1666256189346313, 1.2498911619186401, -0.39732080698013306, -0.220748633146286, 0.6610692143440247, 0.0637362077832222, 0.20892901718616486, -1.060288906097412, 1.4975794553756714, -0.05911370739340782, -1.7431857585906982, 0.1926889568567276, -0.6261370182037354, 1.262861728668213, -2.3450565338134766, 1.001922369003296, 0.4691339433193207, 1.2830421924591064, -0.7507238984107971, 0.06383996456861496, 0.30290162563323975, 1.5331660509109497, 0.057368889451026917, 1.7735950946807861, -1.5230233669281006, 0.009303697384893894, -0.3232728838920593, -0.048992641270160675, -1.4541800022125244, 0.099629245698452, -0.5987293124198914, 0.026035988703370094, -1.3045783042907715, -1.3981571197509766, 1.407550573348999, 1.752949833869934, -1.3454118967056274, -2.0856871604919434, 0.33159950375556946, 0.18458254635334015, 0.30372855067253113, -1.6348028182983398, 0.23585198819637299, -0.016444824635982513, 0.6241836547851562, 0.5265597701072693, 0.3445524275302887, -1.4715473651885986, 0.8211697340011597, 0.14310313761234283, 0.8983418345451355, -0.718834400177002, -1.664244294166565, -1.4030101299285889, 0.36900728940963745, -0.5734978318214417, -1.0803203582763672, -0.863419234752655, 1.0518519878387451, 0.6188783645629883, 0.48367589712142944, 1.6630862951278687, -0.3236927390098572, -0.28375136852264404, 0.7636693716049194, 3.4641478061676025, 1.0358006954193115, 0.1025468111038208, 0.42820432782173157, -1.078490972518921, 1.6753509044647217, 0.3731122314929962, -1.080281138420105, 0.022680282592773438, -0.08567611873149872, 0.9377439618110657, -0.8727587461471558, 0.5065162181854248, 1.4077831506729126, 0.9806623458862305, -1.186525583267212, -1.4287643432617188, -0.6863277554512024, -0.35020726919174194, -1.5574004650115967, -2.0794284343719482, -0.41869017481803894, 2.426746129989624, -0.35041913390159607, 0.5589696168899536, 1.0151058435440063, 0.24660992622375488, 1.168458104133606, -1.3960494995117188, 2.011720895767212, -0.6690654754638672, -0.756720781326294, 0.37630900740623474, 0.06645407527685165, 0.849929928779602, -0.5545939207077026, -0.1220661923289299, 0.3052402138710022, -0.5668506026268005, 0.2524799108505249, -1.7891842126846313, 0.1056997999548912, 0.5553845167160034, -0.3723984360694885, 0.12730155885219574, -0.7469685077667236, 1.8715472221374512, -0.6438722014427185, 1.4069401025772095, -1.21732497215271, -0.13650262355804443, 2.259779691696167, 0.16145700216293335, 0.584824800491333, -0.5826584100723267, -0.27338162064552307, 0.3520034849643707, -0.19982993602752686, 0.3400781750679016, 0.26585763692855835, 0.6972170472145081, -0.08323824405670166, -0.9889702200889587, -0.2708956301212311, -1.7728198766708374, -0.4557746350765228, 0.14143669605255127, 1.1954991817474365, -0.6820216774940491, -0.6907283663749695, 0.6386239528656006, 0.6042023301124573, 0.5859493017196655, -1.4868733882904053, 0.8798994421958923, -0.614764928817749, -0.28328076004981995, 0.11923510581254959, 1.1661980152130127, 0.028207194060087204, 1.6898308992385864, 0.40663912892341614, -0.9143954515457153, 0.3030639886856079, -2.0974273681640625, -0.05832653120160103, 1.128745675086975, 0.4703131914138794, -0.180755153298378, -0.10400108247995377, -0.3057863116264343, 0.44072985649108887, 0.3926839232444763, -1.3571661710739136, -1.3066387176513672, 0.3472275733947754, -0.05475977435708046, 0.18148623406887054, 0.3163566589355469, 1.4820455312728882, 2.101536989212036, -0.0248784851282835, 0.7015435695648193, 1.3851995468139648, -0.7912389636039734, 0.4595508873462677, 1.0486804246902466, -0.5563961863517761, 1.4311256408691406, 1.3126063346862793, 1.7187087535858154, 0.6389843821525574, -0.34629079699516296, 1.7694318294525146, 1.1763174533843994, 1.2219053506851196, -0.20964691042900085, 0.8858795762062073, -1.7357367277145386, -1.2760008573532104, -0.7641594409942627, -1.5889708995819092, -1.1270045042037964, 0.7207010984420776, -0.654366672039032, -0.873995304107666, 0.10642661899328232, -1.6042122840881348, -1.540445327758789, -0.8394558429718018, -1.3684289455413818, -0.2858201563358307, -1.852139949798584, -0.5412111878395081, -0.07041275501251221, -0.22210580110549927, 1.2365907430648804, 1.151370644569397, 1.4532030820846558, 0.31841567158699036, -0.22287240624427795, -0.00470109935849905, -0.5935209393501282, 0.9202055931091309, 1.017098069190979, -0.8837552666664124, 0.5716290473937988, 0.17841988801956177, 2.3075993061065674, -1.1139286756515503, 0.2611958384513855, -0.5484403967857361, -0.8301511406898499, 0.3063609004020691, 0.7574729919433594, 0.4365082383155823, -1.353036642074585, -1.2966253757476807, 0.08451630175113678, 0.2126784324645996, -0.5931499600410461, 1.9642598628997803, 0.3772766888141632, 0.16708087921142578, -1.2763903141021729, -0.25815457105636597, -0.1364811658859253, 0.04336755350232124, -0.019108090549707413, 0.8430420756340027, -1.5983859300613403, -1.5514593124389648, 0.03742478787899017, 0.15301713347434998, -0.685945451259613, -0.3250521123409271, -1.1006686687469482, -0.8365803360939026, 0.33098509907722473, -0.5873080492019653, 1.1095956563949585, -2.471487283706665, 0.37204256653785706, 1.2662255764007568, 0.3405270576477051, 1.5149569511413574, 0.5448746681213379, -0.6976829171180725, -0.2364310473203659, -0.16128012537956238, -0.28104573488235474, 1.124325156211853, 0.7277737855911255, -1.7376662492752075, -0.5834186673164368, 1.484994649887085, 0.4927798807621002, -1.3777822256088257, 0.8346878290176392, 0.3330790102481842, -1.4460036754608154, 1.3270286321640015, 0.5436162352561951, 0.7776654362678528, -0.6600387096405029, -0.8913095593452454], [1.3663887977600098, -0.49041321873664856, -0.28609204292297363, -1.7995798587799072, -0.9678219556808472, -0.1543126106262207, 2.5372064113616943, 0.10431124269962311, 1.6932892799377441, 2.1586761474609375, -0.8256797790527344, -1.6493576765060425, -0.29855775833129883, -0.4501454830169678, -0.16468150913715363, 1.7753970623016357, -0.04523453488945961, -0.9095374345779419, -0.15679806470870972, -0.5337120294570923, 0.022686297073960304, 0.5249703526496887, -0.4783038794994354, -0.04493428021669388, 0.3946721851825714, 0.17328500747680664, 0.548065721988678, -0.9865796566009521, -2.1498003005981445, -0.3404509723186493, -0.41537508368492126, 0.20533114671707153, 0.4712391197681427, 0.4644114077091217, -0.7504148483276367, 1.0901310443878174, 1.298257827758789, 1.5046075582504272, -0.7169440388679504, 0.18264275789260864, -1.1035670042037964, -0.4676916301250458, 0.7163077592849731, -0.5858439207077026, 2.4442248344421387, 1.9412109851837158, 0.5729140043258667, 0.7903618216514587, -1.7234594821929932, 0.0845818892121315, -0.38126319646835327, 0.40678349137306213, -0.4951461851596832, 0.38428744673728943, 0.0384216345846653, 0.4114125669002533, 0.16485074162483215, 0.9446477293968201, -0.5586649179458618, -2.3141355514526367, -0.25397446751594543, -1.002541422843933, 0.5708494782447815, -1.2195813655853271, 0.08041407912969589, 0.6379063725471497, -0.010806722566485405, -0.553743302822113, -1.3965696096420288, -0.4606332778930664, 0.8879982829093933, 0.5385115742683411, 1.236060619354248, -1.653473138809204, -0.7001045346260071, 0.44752997159957886, -0.277113139629364, -0.5648466944694519, 0.37246429920196533, -0.5760623812675476, -0.6212428212165833, 1.0405367612838745, -0.6124023199081421, 0.40129420161247253, -0.2522398829460144, -0.27968162298202515, 1.8263040781021118, -1.602381706237793, -0.38299617171287537, 0.7867258787155151, 0.43047547340393066, -1.234119176864624, -0.7154714465141296, 1.4782829284667969, 0.9719891548156738, -1.4310818910598755, 1.2071717977523804, 1.2600454092025757, 0.9993695020675659, 0.17656835913658142, -1.6999801397323608, 1.5413762331008911, 0.1541386991739273, 0.8816069960594177, 0.7198519706726074, -0.030917279422283173, 2.239244222640991, 1.1092665195465088, 0.2579185664653778, 1.5788229703903198, -1.0160070657730103, 0.22590304911136627, 1.3567359447479248, -0.08911798149347305, 0.003564750077202916, 1.5475132465362549, 0.04642898589372635, 1.4985265731811523, -0.4125274419784546, 1.0353903770446777, 1.1003265380859375, 0.7920097708702087, -0.2187085747718811, -0.9073905944824219, -0.40605428814888, 0.6741654872894287, -1.674608826637268, 0.9188335537910461, 0.6427251100540161, -0.8328598737716675, 0.4279145300388336, -0.5830164551734924, 0.03201628848910332, -1.5278278589248657, -1.459446668624878, -0.6650001406669617, -1.2395892143249512, -1.0909061431884766, 1.2733831405639648, 2.1412112712860107, 0.7929237484931946, -0.5072327256202698, -0.5979358553886414, 0.5645886659622192, -0.3955082595348358, 0.3641199767589569, -1.4296942949295044, -1.709012508392334, -1.14625883102417, 0.9218465089797974, -0.5946372747421265, 1.4098803997039795, 0.8718433976173401, 0.31422021985054016, 0.2610434889793396, -0.5307355523109436, 0.43367069959640503, 1.2864035367965698, 0.21610891819000244, 0.13010723888874054, 1.3704065084457397, 1.5662832260131836, 1.5550858974456787, -0.348092257976532, 0.7857728004455566, 0.810940682888031, -1.1890108585357666, 1.1281977891921997, -0.43796035647392273, -0.6967034339904785, -0.37204205989837646, -1.014232873916626, 0.6262286305427551, 0.03231535106897354, -0.801171064376831, -0.5429462790489197, 1.2756108045578003, 0.113692507147789, 1.6149942874908447, -0.8386784791946411, 1.0843322277069092, -0.5004159212112427, -1.2994403839111328, -0.11386392265558243, 0.9695450067520142, -0.35495805740356445, -0.18796725571155548, -1.0837175846099854, -0.2647801637649536, -1.2286854982376099, 0.5091686248779297, 0.481303334236145, -1.0575058460235596, -0.5432186126708984, 1.1699568033218384, 0.38723817467689514, -0.9783145189285278, 0.13402612507343292, -1.2638072967529297, 0.39505326747894287, 0.21589289605617523, 0.3402571976184845, 0.5086653828620911, -0.9831270575523376, 1.240635633468628, -0.6621747016906738, 1.0746039152145386, -0.5768536329269409, -0.684016764163971, 2.0348422527313232, 0.35524532198905945, 0.37462717294692993, 0.44585511088371277, -0.48974937200546265, 1.039589285850525, 0.13918402791023254, 1.5392613410949707, 0.4891672730445862, 0.07678312063217163, -1.348924994468689, 1.1843942403793335, -2.2137274742126465, -1.0751559734344482, 0.6538658738136292, -0.6674399971961975, -0.6225378513336182, -1.669059157371521, 2.9650020599365234, -0.3793601393699646, -0.2062164843082428, -1.1399482488632202, -0.4256443679332733, -0.30974081158638, -0.30250611901283264, 1.563206672668457, -0.23263263702392578, -0.47230884432792664, -0.11853946000337601, -0.5472448468208313, 0.6169033646583557, -0.5197402238845825, 0.14702850580215454, -1.3092937469482422, 1.4833431243896484, 1.5041935443878174, -0.7638230323791504, 0.038188718259334564, 1.1426310539245605, -0.10647037625312805, 0.38608744740486145, -1.5645146369934082, -0.9537588357925415, 0.35065340995788574, 0.20037348568439484, -0.13460716605186462, -1.5226908922195435, 0.06662548333406448, 0.6075964570045471, 0.7423283457756042, -1.1168668270111084, -0.21356172859668732, -0.4095431864261627, 0.42692655324935913, 0.5059695839881897, -0.3139182925224304, 1.843975305557251, 1.2120195627212524, -0.970914363861084, 0.40191057324409485, -0.4784371256828308, 0.7619739174842834, -1.5211782455444336, -0.82768714427948, 1.220834732055664, -1.152978539466858, 0.3530464470386505, -1.1178624629974365, -0.6856828927993774, -0.8546743392944336, -1.8816184997558594, 1.2849844694137573, 0.5162333250045776, 1.5298266410827637, -0.832744300365448, -1.4439815282821655, 1.77017080783844, -0.10931909084320068, -0.8749566674232483, -0.6134790182113647, 1.0582749843597412, 0.1700289398431778, -0.46258029341697693, 0.0541641004383564, 0.6426236033439636, 1.0556161403656006, -0.34855857491493225, 0.06998095661401749, -1.0126630067825317, -0.7748839259147644, -0.030828289687633514, 0.7757883071899414, 1.3252575397491455, -0.6712755560874939, -1.1181280612945557, 1.035153865814209, 0.4360525608062744, -0.07131224125623703, 1.1915338039398193, 1.7692755460739136, -0.49738967418670654, -1.171868085861206, -0.20403549075126648, -0.3200300335884094, 1.1979928016662598, -0.5607341527938843, 1.0318390130996704, -0.16868267953395844, -1.199053406715393, 0.42564675211906433, 0.5498674511909485, -0.1356193572282791, -1.1835299730300903, -1.801770806312561, 0.40907415747642517, -0.7473604679107666, 0.15037965774536133, -2.096567392349243, -1.3177977800369263, -0.9233229160308838, 0.24758771061897278, 1.4474537372589111, -1.9516065120697021, -0.23367390036582947, 1.5271108150482178, 1.2686856985092163, -0.1331489384174347, -0.09274226427078247, -0.6625251173973083, -1.1626121997833252, 0.5480953454971313, -0.12369097024202347, 0.5307539701461792, -1.0213943719863892, -0.26766735315322876, 0.24231760203838348, 1.6971129179000854, -1.286932110786438, -0.1685216873884201, 1.4537616968154907, -0.05439015105366707, -0.18247319757938385, 0.12781789898872375, -0.25897732377052307, -0.4447062015533447, 0.4181658625602722, -1.1768571138381958, -1.2676575183868408, -0.12895935773849487, 1.6938564777374268, -0.17861177027225494, -0.5959203839302063, 1.0978959798812866, -0.671271800994873, 0.4029087722301483, -0.14666959643363953, 0.45361581444740295, 1.6105424165725708, 0.5504575967788696, -2.565645933151245, 1.0068657398223877, -1.2889413833618164, -1.5984090566635132, 0.31154483556747437, -0.3218349516391754, -1.288037657737732, 0.5328814387321472, 0.946874737739563, -0.49864912033081055, 1.2687503099441528, 0.9888155460357666, -0.6542324423789978, -0.7530720829963684, -0.0948878824710846, 0.21464072167873383, -0.010693179443478584, 1.1901990175247192, 0.34329983592033386, 0.4251299202442169, 0.6056812405586243, 0.34334394335746765, 1.9282468557357788, 0.8738477230072021, 0.8086056709289551, 0.5000169277191162, 0.6536953449249268, 2.0282974243164062, 1.785240888595581, -0.6756336688995361, -0.3866439461708069, -0.10933564603328705, 0.8415617346763611, -0.9752491116523743, 0.1937459111213684, 0.8869155645370483, 0.14417265355587006, -0.0718274787068367, 2.1365315914154053, -2.3491156101226807, -0.0802595317363739, 0.3892550766468048, -1.5808711051940918, 0.48282214999198914, 0.019482268020510674, 0.46298152208328247, -1.2569445371627808, 1.6875669956207275, -0.7503398060798645, 0.4624428451061249, 1.1571682691574097, -0.2923153340816498, -0.7921459078788757, 1.2445223331451416, 1.3449994325637817, -1.4787354469299316, -1.1433665752410889, 0.090940922498703, 0.33014583587646484, -1.4687893390655518, 1.7939666509628296, -0.4910045862197876, 1.1848559379577637, -0.583794116973877, -0.0932937040925026, -0.21863718330860138, -0.38726919889450073, -1.8520792722702026, -1.7154680490493774, 1.7235971689224243, -1.8363192081451416, 1.2031078338623047, 1.9747124910354614, 0.2862425148487091, 1.366294026374817, 1.2649424076080322, -2.2798678874969482, 0.8692944645881653, -0.9586595296859741, -1.5114569664001465, -0.10221663117408752, 0.20833274722099304, 0.37975725531578064, 1.2086044549942017, 1.2069008350372314, 0.31675246357917786, 0.4325796067714691, 1.9218547344207764, 2.667365074157715, -1.6317020654678345, -0.27978554368019104, 0.21185727417469025, -0.046653810888528824, -0.5877448916435242, 0.9637970924377441, -0.539817750453949, 0.2828198969364166, -0.22887752950191498, 0.28367865085601807, 0.18594999611377716, -1.5826895236968994, 1.8243049383163452, -1.2199441194534302, 0.5212680697441101, -0.37110060453414917, 0.573805570602417, 0.2463608682155609, -1.6192721128463745, -0.5480481386184692, -1.8942078351974487, -0.47220081090927124, -1.816332221031189, -0.5725623965263367, -0.5801787972450256, -1.0723735094070435, 0.9229627251625061, 0.5401759147644043, 0.6392994523048401, 0.5811962485313416, 0.8334060311317444, 0.06406669318675995, -1.265154242515564, -0.7777242660522461, 1.7293370962142944, 0.27237194776535034, -0.9865602254867554, 1.0122206211090088, -0.5059537887573242, 1.0583478212356567, -0.9627213478088379, 1.3239612579345703, -0.544567346572876, 0.7494104504585266, 0.4813396632671356, 1.2419215440750122, -1.1619623899459839, -0.6410136222839355, -0.44134223461151123, 2.165647268295288, 0.6613095998764038, 1.9475585222244263, 0.08583522588014603, -0.6149611473083496, 0.7454373836517334, 1.8433141708374023, 0.44509896636009216, 0.5505170226097107, -0.4454164505004883, -0.840212345123291, 1.419522762298584, 0.6805964708328247, 0.5400230288505554, 1.2247964143753052, 0.9624550938606262, -1.3426824808120728, 1.4724071025848389, 0.5028178691864014, 0.6725429892539978, -1.531463623046875, -2.318734884262085, -0.6216150522232056, -0.6921672224998474, 1.0622435808181763, -0.9853566288948059, -0.08229146897792816, -1.3359192609786987, 0.42888015508651733, 0.3707728683948517, 0.29988446831703186, 0.005227026529610157, -0.7805352210998535, -0.43048015236854553, -0.49489352107048035, -0.3583305776119232, -1.6126421689987183, -0.6509554386138916, 0.19650644063949585, 0.11342068761587143, 0.5352622270584106, -0.4219227135181427, 0.4789193272590637, 0.804853618144989, -0.38698557019233704, 0.5087548494338989, -0.5721164345741272, 0.30382293462753296, 1.2122184038162231, 1.2188353538513184, 0.47296667098999023, -0.3159228265285492, 0.7527352571487427, -0.17030273377895355, 0.0026741125620901585, -1.1176332235336304, 2.0269792079925537, 0.21352800726890564, 0.7353919148445129, -0.9661173224449158, 0.7287086844444275, 0.07525263726711273, 0.12788616120815277, -0.1153637170791626, -0.4727819561958313, -0.6696426272392273, -0.07827187329530716, -0.010319596156477928, -0.3824899196624756, 0.9031097292900085, -0.31593602895736694, -0.10347270965576172, 0.6523929238319397, 0.5740028619766235, -1.271655559539795, 1.683514952659607, 2.384164333343506, -1.4524322748184204, 0.059337105602025986, 0.5207936763763428, -0.5569616556167603, -0.27365246415138245, -1.5226984024047852, 0.7232665419578552, 0.6089757680892944, -0.7673929929733276, -0.5443544387817383, 1.9771544933319092, 0.43094491958618164, -0.4243864119052887, 0.9889762997627258, -0.899287223815918, -0.8164107799530029, 0.6374483108520508, -0.5843485593795776, -0.6185166835784912, -0.716435432434082, -0.3817618191242218, 1.5066617727279663, -0.6863387227058411, 0.803800106048584, 2.2719860076904297, 0.15091344714164734, 0.39704629778862, 1.3968747854232788, 0.993928849697113, -0.0042016576044261456, 1.1507340669631958, 1.355290412902832, -0.6494060754776001, -0.2120041400194168, 1.9354033470153809, 0.2283768653869629, 0.28177547454833984, -0.43632858991622925, -0.8572062253952026, 1.9870463609695435, 0.7653821110725403, 1.3622684478759766, 0.8927273750305176, 0.15326549112796783, 0.7363088130950928, 1.28374183177948, 0.36361104249954224, -0.6237594485282898, -0.6775051951408386, -1.568343162536621, 0.8994956612586975, -1.6367710828781128, -0.015054799616336823, 0.45397505164146423, 1.5682843923568726, -0.4280555844306946, -1.9853273630142212, 0.7995681762695312, -1.1755647659301758, -0.5824706554412842, -1.782051682472229, -1.2367807626724243, 1.419344186782837, -1.0127326250076294, -0.5854437947273254, 0.8872604370117188, -1.3524651527404785, -1.0624548196792603, -1.0187278985977173, -1.533493161201477, 1.4317784309387207, -2.2267563343048096, 1.344067096710205, 0.33955419063568115, 1.1999762058258057, 0.6572446227073669, 0.4359399676322937, 1.7472056150436401, 0.529408872127533, -0.8875709772109985, -0.10899776965379715, -0.8407258987426758, 1.7061129808425903, -0.16930587589740753, -0.4731453061103821, 0.0804935023188591, 0.674972653388977, -0.569869339466095, -0.4747520387172699, 1.1602723598480225, -1.9711549282073975, 1.3645210266113281, 0.29782137274742126, 0.4513874650001526, -1.0472835302352905, 1.4367287158966064, -2.4571757316589355, -0.48327943682670593, -0.5039829611778259, 0.22736026346683502, -1.5064146518707275, 1.3166289329528809, -0.1740177720785141, 0.3166079521179199, -0.048968736082315445, 0.4675125479698181, -0.3413419723510742, -0.2950325906276703, 1.0529747009277344, 1.4638742208480835, -1.0395803451538086, 0.8460210561752319, 0.3477480113506317, 1.729610800743103, -0.48436209559440613, -0.174814835190773, -0.03484649211168289, 1.053890585899353, 1.7793387174606323, 1.298987627029419, 2.3119680881500244, 1.7602368593215942, 0.7446470260620117, 1.3061097860336304, -1.7614836692810059, 0.5201677083969116, 0.7029886245727539, -0.5230216383934021, 0.9927594661712646, 1.2804867029190063, -0.7542728781700134, -1.730597972869873, 1.8818813562393188, -0.010676576755940914, -0.5953961610794067, -0.6863445043563843, -0.3997896611690521, -0.8293396830558777, 0.4722789227962494, 1.2817611694335938, 0.1330077052116394, -0.5755569338798523, -0.14122654497623444, 0.04348815605044365, 0.30546942353248596, -1.135426640510559, 1.046345591545105, -0.04880847781896591, -1.5889747142791748, 0.1897030919790268, -0.6561030745506287, -0.3520239293575287, 1.185681939125061, -0.24719780683517456, 0.09672915935516357, -1.3355484008789062, 0.8284741044044495, 0.7331740260124207, -0.6827389001846313, -0.6550427675247192, -1.265261173248291, -0.5374253392219543, -1.1877745389938354, 1.5720916986465454, 0.09460819512605667, 0.5063860416412354, 1.8292797803878784, -0.17490032315254211, -0.566819965839386, -0.3143079876899719, -0.043234843760728836, 0.08562188595533371, -0.3930814862251282, 1.5430248975753784, -0.35399332642555237, -0.2693383991718292, 0.513221263885498, 0.2674091160297394, -0.7227885127067566, -0.9049042463302612, 1.7362840175628662, -0.45798832178115845, -1.949926495552063, 0.41778692603111267, -0.2317631095647812, 1.7517170906066895, -0.37139448523521423, 0.3315370976924896, 2.007838726043701, 0.3818665146827698, 1.294198751449585, -0.02485356107354164, -0.4761495590209961, 1.9354926347732544, -1.0662035942077637, 0.7452157735824585, -0.7060278654098511, -0.16171948611736298, -0.4439483880996704, -0.8384729623794556, -0.33547282218933105, 0.6242771744728088, -0.8550512194633484, 1.139561653137207, -0.9238882064819336, -1.7948867082595825, 0.7858414649963379, -0.6451807022094727, -0.5248914957046509, 0.7033361196517944, 0.9286017417907715, -2.1399998664855957, -0.8466764688491821, 1.3187357187271118, -0.145467147231102, -1.00913405418396, -0.005038742441684008, -0.14271734654903412, -0.3329571485519409, -0.5304409265518188, -0.7282199859619141, 0.4250025451183319, 1.7308640480041504, 0.39499902725219727, -0.5718848705291748, 0.3945169746875763, -0.42820799350738525, 0.18466594815254211, -0.9189350605010986, 0.680205225944519, -1.6768337488174438, 0.44665950536727905, 1.2959367036819458, -0.43650004267692566, 0.4713404178619385, 0.6064886450767517, 0.8826383948326111, -0.7424934506416321, -0.09323877841234207, -0.19105665385723114, 0.3846135437488556, 0.16205520927906036, -1.3438739776611328, 0.7973020076751709, 0.5460107922554016, -0.24464087188243866, 0.7866713404655457, -1.679104208946228, -0.2996627390384674, 0.7460318803787231, 0.24966204166412354, -0.6148346662521362, 0.44813182950019836, -1.166979193687439, -1.3793537616729736, 0.4984889626502991, -0.017298707738518715, 3.0724706649780273, 0.3667668104171753, 0.9007406234741211, 0.38433054089546204, -1.28714919090271, -3.0058727264404297, -0.24854083359241486, 0.4011555016040802, -0.7192068696022034, 0.3900272250175476, 0.340398907661438, 1.0502312183380127, 0.43763259053230286, 0.6693888306617737, -0.7820321321487427, 0.056342802941799164, -0.5919216871261597, -0.5669302940368652, 0.48723727464675903, -1.8770453929901123, 0.31104394793510437, -0.6294043064117432, -0.2012750804424286, 0.8472291231155396, 0.13971741497516632, -1.0135657787322998, -1.8783206939697266, -0.7261525988578796, -0.15585637092590332, 0.9321449995040894, 0.6036319136619568, -0.7526051998138428, -0.9246525168418884, -0.26909589767456055, -0.3965050280094147, 0.16785356402397156, 1.0634827613830566, 0.49177664518356323, 2.3719687461853027, 1.0668634176254272, -0.5128055810928345, -0.9224693775177002, 1.6641768217086792, -1.2685883045196533, -0.031195668503642082, -0.4064943790435791, -0.2226329743862152, -0.25714781880378723, -1.2234472036361694, 0.2222875952720642, 0.4761836528778076, -0.5512236952781677, 1.6123936176300049, 0.9448448419570923, -1.1988983154296875, 0.010649215430021286, -1.045274019241333, -0.06623009592294693, 2.5559606552124023, 1.1464738845825195, -1.1680597066879272, -0.48098668456077576, -1.7355817556381226, -0.854519248008728, 0.25335583090782166, 0.1142784059047699, -0.2785961925983429, 0.1418895274400711, -0.08455214649438858, -1.6180691719055176, 1.1782654523849487, 1.0464195013046265, -0.15257419645786285, 0.21590039134025574, 1.124320387840271, 1.2337197065353394, -0.21745039522647858, -0.4085853397846222, -0.43125540018081665, -0.8697636723518372, -0.819459855556488, -0.8048401474952698, 0.9095001220703125, 0.7522575855255127, 1.1578525304794312, -2.518702268600464, -1.3076987266540527, -0.4468704164028168, -0.7424525022506714, 1.2387030124664307, -0.030403506010770798, 1.0624607801437378, -0.45096659660339355, -1.6128296852111816, -0.40068915486335754, -1.4277129173278809, -0.25205448269844055, 0.657066822052002, 1.222185730934143, 0.07110149413347244, -1.2377855777740479, -0.8350124359130859, 1.750360131263733, -1.6717212200164795, -1.0329571962356567, 0.4831269383430481, 0.7524953484535217, -0.37636613845825195, 0.14354147017002106, 1.1492630243301392, 0.7359645962715149, 0.8260694742202759, 0.49203070998191833, 0.404652863740921, 2.3298299312591553, -0.784128725528717, -1.0838232040405273, 1.3630207777023315, 1.3139532804489136, 0.9409028887748718, -0.20967024564743042, -0.5673468708992004, 0.44513607025146484, -0.8218368887901306, 1.2728369235992432, -0.01189799141138792, -0.6812477111816406, -0.5072458386421204, 1.4068444967269897, -0.7769784927368164, -0.21274630725383759, -0.63013756275177, -0.25755131244659424, 0.6625476479530334, 0.27433428168296814, 1.6910641193389893, -0.2685438096523285, 0.9234018325805664, -0.6315481662750244, -0.17495445907115936, 1.0213412046432495, 0.31553545594215393, -1.9042571783065796, -1.3141909837722778, 0.1669088751077652, 2.027881145477295, 0.3125324547290802, -0.8953078389167786, -0.3199332058429718, -1.5614192485809326, 0.848824679851532, -0.23693817853927612, -1.1439136266708374, -0.08411192893981934, 0.8642932772636414, 1.4540472030639648, -0.2986399829387665, -0.09457028657197952, 0.2759961783885956, -1.496968150138855, -0.9693707227706909, -0.25300276279449463, -0.2949289381504059, 1.3131054639816284, -0.5488742589950562, 0.5344341993331909, -0.46030986309051514, 0.1942673772573471, -0.0967240110039711, 0.51905357837677, 0.6284148097038269, 1.0551328659057617, -1.1041476726531982, -0.42669621109962463, 0.8949600458145142, 0.07522675395011902, -0.3530735671520233, -1.8257067203521729, 0.9756572246551514, 0.06196591258049011, -1.357424259185791, -2.974745035171509, -0.3363633155822754, 1.3952759504318237], [0.11750958859920502, -0.41643309593200684, 0.5526698231697083, -0.3230167627334595, 0.06895435601472855, -0.7696633338928223, -0.1909196525812149, -0.5420821905136108, 0.9612361192703247, 1.5449888706207275, -1.729152798652649, 0.1128230020403862, -0.06772402673959732, -0.33988890051841736, -0.34531402587890625, -0.0846954956650734, -0.03781147673726082, 1.072325587272644, -1.720374584197998, -1.0194993019104004, 1.1747785806655884, 0.3103557825088501, -0.8358383178710938, 1.727980136871338, 0.6240874528884888, -0.4653328061103821, -1.029583215713501, 0.050235114991664886, -0.029196975752711296, 0.06939151883125305, 0.7223846912384033, 0.4752918481826782, -1.7679975032806396, -0.9794257283210754, -1.1657298803329468, 0.09127462655305862, 0.7740824818611145, -0.46574655175209045, 0.1726115345954895, -1.291594386100769, -1.5354019403457642, 1.6263868808746338, 0.012405182234942913, -0.40296924114227295, -0.5752785205841064, 0.207412451505661, -0.37268054485321045, 0.7894615530967712, 0.19167833030223846, -0.36622002720832825, -0.4877046048641205, 0.2367243468761444, -0.6000500321388245, -2.128335475921631, -0.6779382228851318, 0.34100303053855896, 0.10676170140504837, 1.961962342262268, -0.9545029997825623, 0.303357869386673, -0.4791538715362549, 0.7354409098625183, 0.7436408996582031, -0.4881061017513275, 0.6838401556015015, 0.8651386499404907, 1.3024591207504272, -0.5337249636650085, 1.4198817014694214, 0.28422120213508606, 0.01787405088543892, -0.9694646000862122, 0.07392212748527527, 2.10855770111084, 0.5364227294921875, 0.35750287771224976, -0.22545559704303741, -0.2584105432033539, 0.026009900495409966, -0.8844705820083618, -0.8130553960800171, 0.38951730728149414, -1.3223519325256348, 0.00027959662838838995, 1.6288233995437622, -0.24342629313468933, -1.221193790435791, 0.40737590193748474, -0.18977391719818115, -0.39264628291130066, 1.5652859210968018, -2.21329927444458, 0.1504465639591217, 0.7153846621513367, -0.5098539590835571, -0.4856742322444916, 0.3077355623245239, 0.8169482350349426, -1.5819655656814575, 0.9260982871055603, -0.2730843722820282, 1.0863710641860962, -0.12220988422632217, 0.44131600856781006, 1.1437939405441284, -0.2960403263568878, 0.8569236993789673, -1.688248872756958, -0.3233117163181305, -0.6095673441886902, 0.9105345606803894, -1.2417478561401367, 1.0814907550811768, -1.2843362092971802, -0.18855009973049164, -1.4612635374069214, 0.2784818708896637, -0.5176586508750916, -0.7248874306678772, -0.8905576467514038, -0.5842161774635315, 0.2993495762348175, -0.030146392062306404, 1.1715741157531738, 0.6253534555435181, 1.199118733406067, 1.5248535871505737, 0.32432156801223755, -1.5202531814575195, -1.2948071956634521, -0.39867281913757324, -0.12047524750232697, -0.5821143984794617, -1.2880622148513794, 1.0085066556930542, -0.02778703160583973, -0.7862749695777893, 1.31500244140625, 0.7725807428359985, -0.6017012596130371, 0.6704549789428711, -0.2854352593421936, 0.8395494818687439, 0.433686226606369, 0.05364987626671791, 0.9816177487373352, -1.0412636995315552, -1.338030219078064, 1.2304924726486206, 0.06917604804039001, 0.7940336465835571, 2.052490711212158, 0.7910952568054199, -2.053581714630127, -0.08337125182151794, 1.996207594871521, 1.1841976642608643, 0.503376305103302, -0.013833827339112759, 0.6492603421211243, 0.8101179003715515, 1.4994969367980957, 0.7374950647354126, -0.05669545382261276, 1.3706690073013306, 0.8064221739768982, -0.9611667990684509, -0.1353832334280014, -1.123189091682434, -0.700738251209259, -0.622043251991272, 1.8297219276428223, -0.4332311749458313, 1.1866626739501953, -1.1394710540771484, -1.4801335334777832, -1.214355230331421, -0.653624415397644, 0.3279912769794464, 0.29780563712120056, -1.2124106884002686, 0.8601071238517761, -0.8582544922828674, 0.15471160411834717, 0.47420451045036316, 1.437487244606018, -0.8067119121551514, 0.29856547713279724, 0.5250150561332703, 0.8054367899894714, -0.9602962732315063, 0.11684169620275497, -0.5942289233207703, -0.34556248784065247, -0.4715808629989624, -1.2755720615386963, -0.9155539870262146, 0.8423966765403748, 0.6496835947036743, -0.23500767350196838, -0.6390907764434814, 1.0595741271972656, 0.08243583142757416, 0.22684027254581451, 0.28800058364868164, -1.2634042501449585, -1.539424180984497, -0.00813943613320589, 0.8508561849594116, 1.231136679649353, 0.1666157841682434, 1.0675411224365234, 0.41324684023857117, -0.1793081909418106, -0.7769438624382019, 0.6807025074958801, 0.14081251621246338, -1.388911485671997, -0.9359822869300842, -0.6716947555541992, 1.158355474472046, -0.6197055578231812, -2.557802200317383, 1.9450650215148926, 0.004718495067209005, 2.932126760482788, -0.022652506828308105, -1.7248800992965698, 0.5309748649597168, 1.0481706857681274, 0.12737539410591125, 0.31779780983924866, -0.006962680257856846, -1.9560809135437012, -1.9307303428649902, 0.2606625258922577, 1.6961281299591064, -0.6554398536682129, 2.7119555473327637, 0.37559494376182556, 0.3334014415740967, -0.9487496018409729, -0.2644904851913452, -0.5568814873695374, 1.2460166215896606, 1.102473258972168, 1.3006881475448608, -0.6495299339294434, -1.31844961643219, -0.7543182373046875, 0.5514945387840271, -1.1216777563095093, 1.6792771816253662, 0.9127811789512634, 0.7610905170440674, -0.7215632796287537, -0.6308131814002991, 0.2728729546070099, -1.664501667022705, -0.5815784931182861, -1.3508834838867188, 1.1890486478805542, -0.11932747066020966, -0.4328992962837219, 1.121281623840332, -0.7389560341835022, -0.2644059956073761, 1.1160509586334229, -0.40531960129737854, -0.08123606443405151, 0.7190513014793396, -0.0276480745524168, 0.061229076236486435, -0.6029980778694153, 0.13374872505664825, -1.0929573774337769, -1.0277602672576904, -0.6314364075660706, -0.659784734249115, -0.0648631900548935, -1.3991341590881348, -0.022343892604112625, -0.10844483971595764, -1.435741901397705, 0.09263767302036285, -0.1694273203611374, 0.3826242983341217, 1.5861362218856812, -0.39351457357406616, -1.482021450996399, 0.9953539371490479, -0.3251676559448242, -0.766997754573822, 0.7591843008995056, 0.49445152282714844, -0.008051075972616673, 0.19699692726135254, -3.2654521465301514, 0.41076013445854187, 0.1131434291601181, -0.054724495857954025, 1.4028033018112183, 0.8575824499130249, 0.39418116211891174, 0.8284252882003784, -0.6571720838546753, 0.5083884000778198, -1.6722540855407715, -0.02846527099609375, -0.7225876450538635, -0.9556970596313477, -0.7869030833244324, 1.947178840637207, 0.3703581392765045, 0.7783710956573486, 0.4691373407840729, -0.717567503452301, -0.2761378586292267, 1.4186532497406006, -0.5637491345405579, -0.9864826798439026, 0.1323258876800537, -0.6790005564689636, -1.6858464479446411, -1.1931145191192627, 0.4518663287162781, 0.33128875494003296, -0.703353226184845, 1.5207911729812622, 0.0045226626098155975, 0.03305942565202713, 0.5440675020217896, 0.8401682376861572, -1.9157835245132446, -2.757107734680176, 0.21908138692378998, 0.29420673847198486, -0.19180089235305786, 1.884846806526184, -0.09185993671417236, -1.68852698802948, -0.7638227939605713, -1.6794553995132446, 0.4599168598651886, -0.7622492909431458, -0.3274032473564148, 1.2132620811462402, 0.32759320735931396, 1.0777301788330078, -0.3182344436645508, 0.7737346887588501, -0.38933849334716797, 0.3460039794445038, 2.479295015335083, 0.5842316746711731, -0.8787344098091125, 0.9117450714111328, -0.18795931339263916, 0.5119569897651672, 0.44630345702171326, -0.558490514755249, -0.12771227955818176, -0.9388042688369751, -0.4750913679599762, 0.38138964772224426, 0.006518855690956116, -0.40795204043388367, -0.30124631524086, -0.715838611125946, -0.6072723865509033, -2.5005292892456055, 0.2722366154193878, -2.2272677421569824, 0.796928346157074, 0.8966112732887268, 0.7633277773857117, 0.3138536512851715, 0.8622369170188904, -0.7798682451248169, -2.7181525230407715, -0.960835874080658, -1.628862738609314, 1.2052081823349, -0.4021279215812683, 1.0273927450180054, -0.03908942639827728, -0.7136136889457703, -1.4668382406234741, 0.1485961675643921, 0.23268736898899078, -1.0586328506469727, -0.7146535515785217, -0.9296681880950928, 0.7648142576217651, 1.8199119567871094, -0.5911995768547058, -1.160949945449829, -0.7251794338226318, 1.0204472541809082, 0.33150047063827515, -0.02124110609292984, 0.19277064502239227, -0.9722696542739868, -0.24728815257549286, 1.7278209924697876, -1.038101315498352, -0.6692100167274475, 0.7188843488693237, 0.6850536465644836, 0.6469423174858093, 0.015655647963285446, 1.8951388597488403, 0.05735302343964577, 1.1236467361450195, -0.07701322436332703, -0.8274561166763306, 1.0171153545379639, -1.485169529914856, -0.1366109400987625, 0.6092283725738525, 0.4573509395122528, 0.16009598970413208, 0.6048946976661682, -0.004909425973892212, 1.403106689453125, -0.09656602889299393, -2.081033706665039, -0.45447272062301636, -0.4546468257904053, -0.9527480006217957, -0.44601842761039734, -0.8983364701271057, 0.7516916394233704, -1.6465795040130615, 0.08939938992261887, -0.2129366248846054, -0.8194650411605835, -0.37709376215934753, -0.7913528680801392, -0.3717568516731262, 1.1193405389785767, 0.6979407668113708, 0.5865720510482788, 0.0750918909907341, 1.1235040426254272, 1.500489354133606, -2.035506010055542, 0.09642789512872696, 0.1526404321193695, -1.9127486944198608, -2.4363303184509277, -1.9724750518798828, -1.130817174911499, 0.34584537148475647, -0.7587916254997253, -1.5165040493011475, 0.26066091656684875, 0.3191511034965515, 0.47403913736343384, -0.7872726321220398, 0.41200903058052063, -0.43521085381507874, -1.028968095779419, -0.1278345137834549, -1.427950143814087, -0.9194779396057129, 1.4701135158538818, 1.0880241394042969, 1.2246543169021606, -0.06667257100343704, -0.9474838972091675, -0.4721842110157013, 1.5169568061828613, 0.8414503931999207, -0.2426174432039261, 1.442826271057129, 0.11466646194458008, 0.10685574263334274, -1.8330752849578857, -1.6832020282745361, 1.116690754890442, -0.9163978099822998, -0.04648852348327637, -1.8382645845413208, 0.02731490507721901, -0.9303791522979736, -1.7415190935134888, -1.2759695053100586, -0.45716384053230286, -0.8448562622070312, -1.2207963466644287, -0.04480770230293274, 0.6788227558135986, -1.3431965112686157, -1.36223566532135, 1.734802007675171, -0.5327318906784058, 1.4829421043395996, -0.8424023985862732, -0.38493821024894714, -0.24846288561820984, -0.23053018748760223, 0.21508973836898804, -0.48229074478149414, -2.5099093914031982, 1.42219877243042, 0.1458905190229416, 1.0923992395401, 1.8202530145645142, -0.3391711413860321, 0.33145231008529663, -0.40650632977485657, 0.3655746877193451, -0.11000730842351913, 0.7885828614234924, -1.2746100425720215, -0.3470766842365265, -0.11871299147605896, 0.9610483646392822, 0.03536110743880272, -0.4817855954170227, 0.7814399003982544, 0.6428152322769165, -0.6812275648117065, 0.5286343097686768, 1.231680154800415, 0.3428085744380951, -0.0770154595375061, -1.0521583557128906, -0.16014434397220612, 0.8537796139717102, 0.4242022931575775, -0.460124671459198, 1.0031919479370117, 0.45304837822914124, -0.17576852440834045, -1.9800347089767456, -0.2691816985607147, 1.9707684516906738, 0.3030399680137634, -0.019711004570126534, -1.190492868423462, -0.3633497953414917, 0.4212518334388733, 1.7960902452468872, 0.5521427392959595, 0.9951742887496948, -0.9066327214241028, -0.6023097634315491, 0.2359655797481537, -1.8013873100280762, -1.0241661071777344, 0.019409408792853355, -1.6995865106582642, 1.4864712953567505, -0.23952555656433105, -1.2804334163665771, -0.8572077751159668, 1.3705251216888428, 0.7415251731872559, 2.0110764503479004, 0.5555970072746277, -2.0466349124908447, 0.23703734576702118, -0.9472398161888123, -0.008735045790672302, 1.0965105295181274, -1.1325026750564575, -0.045044053345918655, -1.1998958587646484, -0.9870547652244568, 0.4890037477016449, 0.036286287009716034, -1.1572011709213257, 0.8325169086456299, 0.7482746243476868, 0.26538971066474915, -1.273248553276062, 1.0974202156066895, -0.6947013139724731, -2.388543128967285, 0.6625064015388489, 0.7690017223358154, -1.1936204433441162, -0.04284806177020073, 0.10244368761777878, 1.8207488059997559, 0.7183255553245544, -0.01636291667819023, 0.17576734721660614, 0.45746514201164246, 1.1640199422836304, -1.113433837890625, -1.2940895557403564, 0.3019739091396332, 0.16843783855438232, 0.5734038949012756, 2.635472059249878, -1.1845487356185913, -0.31355395913124084, -0.46177026629447937, -1.5485173463821411, 2.291184902191162, -1.0720784664154053, -0.9226707816123962, 1.285753846168518, -1.0218071937561035, -1.657618761062622, -1.1333322525024414, -0.771362841129303, -0.5465050935745239, -0.7824440598487854, -0.6869722008705139, 0.33320754766464233, 0.0675060898065567, 2.0953774452209473, -0.38107243180274963, 1.0694657564163208, -1.330613374710083, -0.8089481592178345, -0.08898945897817612, 0.05855945125222206, -0.2646216154098511, 0.32354599237442017, -0.13805410265922546, -0.04225049540400505, -0.7460260391235352, -0.14332742989063263, -0.0040086605586111546, 1.285169005393982, 1.2227824926376343, 1.178685188293457, 1.9544379711151123, -0.5262928009033203, -0.9670608639717102, -0.2541307806968689, 0.46419811248779297, -1.3659619092941284, 0.7666414380073547, -0.7581281065940857, -0.041811175644397736, 0.7079651355743408, 1.1484994888305664, 0.5752113461494446, -0.19669616222381592, -0.9561818838119507, 1.7706362009048462, -0.3738749027252197, 0.5621688365936279, -0.616838276386261, -0.8863285779953003, -2.777554988861084, -2.5438668727874756, -0.02785375900566578, 0.6413784623146057, -0.0633891224861145, 0.20092496275901794, -0.530063807964325, -0.18665950000286102, 0.500586748123169, -1.415650725364685, -0.6313944458961487, 0.27274179458618164, 0.8027076721191406, -1.510532021522522, -1.722280740737915, -2.8241214752197266, 0.4390396475791931, 0.7573359608650208, -1.3097951412200928, -0.16798779368400574, -0.43997621536254883, 1.4096040725708008, 0.5727370977401733, -0.8136336803436279, -0.7364294528961182, 0.264845609664917, -0.5530834197998047, -0.8250987529754639, 0.2660159766674042, -0.36223727464675903, 0.6874346733093262, 0.5219268798828125, 0.44804614782333374, -1.041133165359497, 0.2517940402030945, -0.04371398687362671, -0.8544139862060547, -0.09928672015666962, 0.035243961960077286, -0.9339331984519958, 0.37366607785224915, -0.3032439649105072, -1.7710444927215576, 0.4074692726135254, 0.6789612770080566, 1.4214454889297485, -0.44995027780532837, -0.6461809277534485, -1.190705418586731, 0.13256214559078217, -1.6326298713684082, 0.05794578418135643, 0.41786840558052063, 0.6390393376350403, -0.6639320254325867, 0.3925820291042328, 2.092522144317627, -0.45401355624198914, -1.2171071767807007, -0.5960983633995056, 0.40553662180900574, 0.005468151066452265, 1.1338917016983032, 0.16961541771888733, 0.4044022262096405, 0.47486579418182373, 0.9829097390174866, 0.2515518367290497, 0.1406984180212021, -0.23483867943286896, -0.3999982476234436, 0.7532023191452026, 1.107168197631836, 1.3137125968933105, 0.432920902967453, -1.3588111400604248, 0.259876549243927, 0.9531997442245483, -0.14567701518535614, -1.4593148231506348, -0.22164395451545715, -0.5622814297676086, -0.20739373564720154, 1.0979658365249634, -2.6468594074249268, 1.0278065204620361, -0.6582180261611938, 0.6995680332183838, -1.5166081190109253, -0.880779504776001, -0.33687320351600647, -0.23998774588108063, -0.8591152429580688, 0.5110008716583252, 1.2588481903076172, -0.633797824382782, 0.10077095031738281, -0.005047055892646313, -0.5257865786552429, -0.27971404790878296, 1.3238521814346313, -1.8740143775939941, 0.26337021589279175, -0.19691503047943115, -1.2968820333480835, 0.172714963555336, -0.7780075073242188, -1.3180211782455444, -0.5830596685409546, 0.37674224376678467, -1.1621499061584473, 0.17895521223545074, 0.5462853908538818, -0.375018835067749, -0.2124018520116806, 1.871724009513855, 0.6108669638633728, -0.007621191907674074, 0.32900485396385193, -0.7189802527427673, -0.696164608001709, 1.3427947759628296, -0.27638548612594604, -0.47557884454727173, 1.2754175662994385, 0.815701425075531, 0.8638466000556946, -0.17455384135246277, 1.002685785293579, -1.0610431432724, 0.22919780015945435, 1.8109455108642578, 1.4613075256347656, -1.3658510446548462, 0.20547719299793243, -1.7163276672363281, -2.5999436378479004, 0.61700838804245, 0.3678072988986969, -0.2667335271835327, -1.3674259185791016, -0.22423985600471497, -1.1634705066680908, 0.5316984057426453, -0.08853704482316971, 0.7572404146194458, -1.430184006690979, -0.7490510940551758, -0.9688675403594971, 0.2298608124256134, 1.15583074092865, -0.2675730288028717, -0.8757453560829163, 1.4681061506271362, 0.5529170036315918, 0.04862836375832558, -0.7791499495506287, -0.5709910988807678, 0.14340665936470032, -1.0386865139007568, 2.293576240539551, -0.03589494526386261, 0.024353522807359695, -0.21037058532238007, 1.02790367603302, 0.44018039107322693, 0.6173830628395081, 1.7275694608688354, 0.26851344108581543, 0.9043458700180054, -0.3299788534641266, -0.6574367880821228, 0.8793547749519348, 0.21849007904529572, 0.539743185043335, 1.4411066770553589, 2.0090949535369873, -0.9358434677124023, -0.006736156065016985, 0.7213990688323975, -0.5985062718391418, 0.005898670759052038, 2.064079761505127, 0.0689292773604393, -0.3456643223762512, -1.3522474765777588, -0.7535269856452942, 1.3179984092712402, 1.0672645568847656, -1.125569224357605, 1.9746273756027222, 0.08742498606443405, 0.42895668745040894, 0.3104877769947052, 0.11576779186725616, 2.2037365436553955, 1.368094563484192, -0.9287671446800232, -0.49937382340431213, -0.08521172404289246, -0.33087942004203796, 3.392423152923584, -0.16355720162391663, -0.8860892653465271, -0.7288775444030762, -0.3143267035484314, 0.23266710340976715, 0.8344168663024902, -1.3639565706253052, -0.12521204352378845, 0.372498482465744, -0.07165303826332092, 0.700221836566925, 0.5740336179733276, 0.33829164505004883, 0.523904025554657, -1.2797980308532715, -1.5906007289886475, 1.823769211769104, 0.08418919146060944, -0.4125732183456421, -1.2251622676849365, 1.03944730758667, -0.9222158193588257, 1.299442172050476, 0.024310402572155, 0.9092802405357361, 0.2262551486492157, -1.1611496210098267, 0.30736055970191956, -1.6013920307159424, 0.8207414746284485, -1.3074649572372437, 0.17134566605091095, 0.5358366966247559, -1.026929497718811, -1.0178067684173584, 1.4800447225570679, -0.649634063243866, 1.2459301948547363, -0.6504234671592712, 0.39070141315460205, -2.246370792388916, 0.5098667740821838, 0.26018616557121277, -0.10144609212875366, 1.3026052713394165, -1.8866409063339233, -1.2882685661315918, -0.2700941860675812, -1.1807392835617065, 0.40213561058044434, 0.3050144910812378, -0.6510041356086731, -0.8387637138366699, 1.4855819940567017, 1.1350243091583252, -1.1454212665557861, 1.592500925064087, 1.5105804204940796, -1.608763575553894, -1.3038238286972046, -0.36168572306632996, -0.1408330798149109, -1.1727932691574097, 0.7278301119804382, -0.7404510974884033, -0.35796990990638733, -0.7639370560646057, 0.7236850261688232, -1.7173726558685303, -1.3943710327148438, -1.4444751739501953, 0.4736963212490082, 0.8878257870674133, 2.1680853366851807, -0.7739218473434448, 1.324963927268982, 0.8239639401435852, -0.304362028837204, -1.021225094795227, 0.042930468916893005, 0.15674184262752533, -1.069010853767395, 0.21490377187728882, 0.6447888016700745, -0.07798425108194351, -0.7153542637825012, -0.35182347893714905, 0.02470039762556553, -0.9025241136550903, -0.10012736916542053, -1.2953966856002808, 0.61342453956604, -1.0430408716201782, -0.009867516346275806, -0.21666495501995087, 0.25652435421943665, 0.3041907250881195, -0.7963508367538452, 1.4698963165283203, 1.0860611200332642, 1.493774652481079, -1.8435298204421997, -2.116231679916382, -1.2720658779144287, -0.8056808710098267, -0.345146119594574, -0.7168503999710083, 0.8834805488586426, -1.7235156297683716, 1.4959036111831665, 0.24336352944374084, -0.2626645863056183, 0.22009380161762238, -2.270024299621582, 0.6578972935676575, -1.8813400268554688, 1.55558180809021, -0.955052375793457, -0.5561625957489014, -0.7615912556648254, -0.6289821863174438, -1.6894720792770386, 0.905601978302002, 0.08215444535017014, 1.1841144561767578, 1.4551403522491455, 0.9458448886871338, -0.32392773032188416, -0.6125828623771667, -1.3189077377319336, -0.3193484842777252, 0.8231310248374939, -0.6841942667961121, 0.5934131145477295, -0.06917177885770798, 1.0623607635498047, 0.2911146581172943, 0.5464370846748352, 0.9367876648902893, 1.1171743869781494, -1.4853425025939941, -0.5616728663444519, -1.6243517398834229, -0.0704118087887764, 0.2188013643026352, -0.7386821508407593, -1.0404220819473267, -0.4901107847690582, -1.2626198530197144, 0.7472874522209167, -1.3448606729507446, 0.06777720153331757, -0.560853898525238, -0.8763121962547302, -0.26806914806365967, -0.04124696925282478, -0.6512101292610168, -1.5948193073272705, 2.346808671951294, -1.3617453575134277, 1.1971664428710938, 0.10064425319433212, -0.4600400924682617, -0.8170013427734375, -0.46935585141181946, 0.40191352367401123, -1.0347516536712646, 0.5171201825141907, -0.9071038365364075, -1.300801396369934, 0.9768791794776917], [-0.8747545480728149, 1.3875724077224731, -0.760206937789917, 0.5852810740470886, -0.06094348430633545, -0.7668115496635437, -1.0822052955627441, -0.5020468831062317, -0.9229648113250732, 1.2511016130447388, 0.2623802721500397, -0.9358639717102051, 1.1233447790145874, -0.34191909432411194, 0.5626047253608704, 2.3658242225646973, -1.2761006355285645, 0.2880649268627167, -1.1280313730239868, 1.584692358970642, 0.8830364942550659, 0.2112164944410324, 0.26355552673339844, 1.519538164138794, 1.2342467308044434, 0.5804978609085083, 0.37621694803237915, -0.9127801060676575, 0.8995380997657776, -2.362840175628662, 1.0308564901351929, -0.39799198508262634, 0.31166911125183105, 0.5750759840011597, 0.9362038969993591, -0.06016295775771141, -0.46105170249938965, 0.316904753446579, -0.6097708940505981, 1.767608404159546, 0.7924931049346924, -1.0494749546051025, 1.8728022575378418, 1.9543935060501099, 0.025967827066779137, -0.20510166883468628, 0.041628412902355194, -0.18424513936042786, 0.38526272773742676, -0.7466920018196106, -0.6386979818344116, -0.610632061958313, 1.0746415853500366, -0.47448211908340454, -1.0717452764511108, 0.8724899291992188, 1.2101131677627563, -0.536651074886322, -0.01500792521983385, -0.24518312513828278, 1.8947083950042725, -1.7244691848754883, 1.7548832893371582, 0.35370439291000366, 0.6322984099388123, -0.30862560868263245, 0.5895819067955017, -0.7529811263084412, 0.6062012314796448, 1.9041290283203125, -0.9528257250785828, 0.4446682631969452, 0.7099300622940063, -0.1132182627916336, -0.847722053527832, 2.3912742137908936, 0.20153746008872986, -0.4679016172885895, -1.181620717048645, 0.24032878875732422, -0.9256681799888611, 1.5401464700698853, -0.04824192821979523, -1.430894136428833, 0.630881667137146, 0.9149234294891357, 1.1508053541183472, -1.4861912727355957, -0.7666831016540527, 1.5152753591537476, -0.013380262069404125, 0.5356420874595642, -1.128692865371704, 1.1715608835220337, 0.219683438539505, -0.881172239780426, -1.340672254562378, -0.1609257161617279, -0.9724031090736389, -0.07765815407037735, -1.1064437627792358, 1.1484076976776123, 2.4622550010681152, -0.05683164298534393, -1.0355491638183594, 0.0983874499797821, -0.8803247213363647, 0.5059389472007751, 0.47489112615585327, 0.45518961548805237, 0.62809157371521, 0.32509663701057434, 0.09290072321891785, -0.16996005177497864, -0.2724940776824951, -0.1362859308719635, 0.025179291144013405, -1.5926527976989746, 0.06058501824736595, 2.2186522483825684, -2.685575485229492, 0.10630035400390625, 0.12545642256736755, -0.0264061838388443, 0.7255575060844421, 1.6413753032684326, -0.9404972195625305, -0.8612772822380066, -1.7210125923156738, 1.7068458795547485, -0.06118198111653328, 1.213815450668335, -0.10701265931129456, 1.3002437353134155, 0.18515773117542267, -1.1246874332427979, -0.6127099394798279, -0.5220829844474792, -0.9254865646362305, 1.2858394384384155, -1.2850435972213745, -0.11209571361541748, -1.3090364933013916, -0.0643698051571846, 0.6681123375892639, 1.371201992034912, 0.6682981848716736, 1.0172940492630005, 0.6893962025642395, -0.9181392788887024, 0.5164574980735779, -0.9074323773384094, 0.5259718298912048, 0.057072486728429794, 0.5095986127853394, 2.075299024581909, 0.9520081877708435, -0.7721478343009949, -2.3602633476257324, -2.994791269302368, -0.9864760041236877, -1.5051816701889038, -1.476469874382019, 0.45506930351257324, 0.06065215542912483, -0.007016061805188656, 0.2293369174003601, -1.8331433534622192, -0.44502171874046326, -0.06250473111867905, 0.7197323441505432, 0.2666204571723938, -0.9313527345657349, 1.430366039276123, -0.8835177421569824, 1.0141067504882812, 0.19943928718566895, -0.8169988989830017, 0.7863087058067322, -0.20635059475898743, -0.7478336095809937, -0.12699870765209198, 0.8923448920249939, -1.5401521921157837, -0.6375636458396912, 1.2317781448364258, 0.3971763253211975, -0.14848698675632477, -0.14656054973602295, -0.4463959336280823, 1.1966989040374756, -0.1010173037648201, 0.9864935278892517, 1.4559823274612427, 1.4000142812728882, 0.08902031183242798, 0.7850005030632019, -1.7868659496307373, 0.9383736252784729, 0.4404984414577484, 0.6222499012947083, -0.19105584919452667, 0.5506583452224731, -0.03390955924987793, 0.26021507382392883, 0.2866870164871216, -0.046866483986377716, 0.8197852969169617, 1.0837013721466064, -0.3747790455818176, 2.925586223602295, -0.4268299639225006, 0.013274005614221096, -1.3176345825195312, -0.28599846363067627, 0.6172040700912476, -0.2780883312225342, 0.032743118703365326, -0.27291956543922424, 0.8788900971412659, -0.2782142460346222, -0.28415587544441223, -1.3367397785186768, 0.4813810884952545, 0.3054259717464447, -0.3508683741092682, 1.2256020307540894, 1.7464890480041504, -0.8426297307014465, -0.5114550590515137, 1.5248216390609741, 0.9509187936782837, -0.2802427113056183, 1.6283084154129028, 0.6338545680046082, 1.0704883337020874, -0.9475610852241516, 1.5686671733856201, 0.8975831866264343, -0.9328731298446655, -0.9249297380447388, -0.7326732277870178, -0.08326420933008194, 0.4564801752567291, 0.08815103769302368, -0.7071037888526917, 0.37270116806030273, 0.2165977507829666, 0.3219630718231201, -0.9905929565429688, 1.502786636352539, -1.4037303924560547, 0.8770338296890259, 0.3749374747276306, 0.5196691751480103, -0.04828966036438942, -0.06310253590345383, 0.8554131984710693, -0.3211804926395416, -0.08230847120285034, 0.8433852195739746, 0.7741483449935913, 1.1427189111709595, 0.38524919748306274, -0.7847491502761841, 1.1762776374816895, -0.7536709308624268, -0.2451685220003128, 0.6294745802879333, 0.2683326005935669, -0.6922480463981628, -0.8383857011795044, 1.1036322116851807, 0.367890864610672, -1.1060330867767334, 0.43118616938591003, 0.6384180784225464, -0.6681389808654785, 0.21102595329284668, 1.4818885326385498, 0.20923513174057007, 0.7596693634986877, 1.2850701808929443, -0.0650654137134552, -1.6098860502243042, -0.6909204125404358, 0.11476220935583115, 0.06338690221309662, -1.4430344104766846, -1.671738624572754, 0.7757409811019897, 0.20160357654094696, -0.32995933294296265, 0.936894416809082, -0.1307409256696701, -0.3314453363418579, -0.7070745825767517, 1.074885368347168, -0.06207822635769844, -0.17285779118537903, 0.0014091350603848696, 1.0749578475952148, -1.0346583127975464, 0.4236152768135071, 1.8270063400268555, 1.1411573886871338, 0.2597067058086395, 0.6679722666740417, 0.47494035959243774, 0.42765647172927856, -0.4328227937221527, 0.2907724678516388, 0.60410076379776, 1.1215633153915405, 1.4345204830169678, -0.10681156069040298, 1.1495890617370605, -0.4372011125087738, 2.1220247745513916, -1.165500521659851, 1.9941574335098267, -0.0945654958486557, -0.22169533371925354, 0.6409758925437927, -0.45855650305747986, -1.1510307788848877, 0.3499290347099304, 0.06567826867103577, 0.34660208225250244, 1.3823864459991455, -0.20881333947181702, -0.4744657874107361, 0.24652500450611115, -0.14417827129364014, -0.9759591817855835, 1.1724361181259155, -0.6258524060249329, -0.13389432430267334, -0.27868059277534485, -0.7790951728820801, 0.9624111652374268, -0.755929172039032, -0.7363184094429016, -2.636000633239746, -0.06492264568805695, 0.48005080223083496, 0.001958116190508008, 0.8128951191902161, 1.8403847217559814, 0.14510297775268555, 0.4651823341846466, -0.05786896124482155, -1.819638967514038, 0.24673067033290863, 0.9517995119094849, 1.252265453338623, 1.1714510917663574, 0.9071006178855896, 0.8821678757667542, -1.2768242359161377, 0.1347278207540512, -0.6613291501998901, 1.1029645204544067, -0.2804192900657654, -0.21208472549915314, 0.8392134308815002, -0.8286234140396118, 0.7069423198699951, -1.38334059715271, -0.35148316621780396, -0.5965331792831421, 1.2605042457580566, 0.2749083340167999, -2.8789381980895996, 0.03018762730062008, -1.17335844039917, -1.3932323455810547, 1.566687822341919, 0.22118037939071655, -0.5261078476905823, 0.7147639989852905, -0.3441520929336548, -0.1039067730307579, -0.4016507863998413, -0.7470778822898865, -0.4996143877506256, -0.03317422419786453, -0.007878010161221027, 0.2876182496547699, 1.2455344200134277, 0.3419061601161957, -0.1811109036207199, -1.2957396507263184, -1.8920860290527344, -0.05340132489800453, -0.21968941390514374, -1.1228049993515015, -0.5401561260223389, 1.4049742221832275, 1.8262819051742554, -0.4418805241584778, -0.012834570370614529, -1.3047531843185425, -1.2635854482650757, -1.7586268186569214, -2.3415844440460205, 0.25609105825424194, 0.019249264150857925, 0.443803608417511, -0.7232273817062378, -0.2490280717611313, 0.8840412497520447, -0.19597691297531128, 1.953969955444336, 1.6882554292678833, 0.2782498300075531, -0.9233392477035522, -0.04202447086572647, 1.6231968402862549, -0.09216970205307007, -0.513535737991333, 2.4443912506103516, -1.3624049425125122, -1.1034631729125977, 0.8897291421890259, -0.03357581049203873, -0.5839148163795471, -0.3497824966907501, 1.067419171333313, -0.9686858057975769, -0.3288586139678955, 0.871286928653717, -0.5952680110931396, 0.2644917964935303, -1.6713629961013794, 0.8724543452262878, -1.1461989879608154, -0.97926265001297, 0.2762465178966522, 0.567374050617218, -0.7393602132797241, -1.531611680984497, -1.2199839353561401, -1.1387509107589722, 0.35820823907852173, -0.5894953608512878, 1.7108917236328125, -0.44780489802360535, 1.054300308227539, 0.542363166809082, 0.23646317422389984, -0.5799014568328857, 0.5312538743019104, 1.5060875415802002, -0.6983558535575867, -1.3941538333892822, 1.6805002689361572, 0.8282816410064697, 1.025533676147461, 2.229994773864746, 0.43142110109329224, -1.586456298828125, 0.061179958283901215, 1.009225606918335, 0.07444348186254501, 0.20667725801467896, -1.0460044145584106, 1.9267349243164062, 2.9008381366729736, 1.349340796470642, -1.3977677822113037, 0.24265339970588684, -1.0686010122299194, 1.3163453340530396, 1.7580132484436035, -0.03700394928455353, -0.5327125787734985, -0.9870667457580566, 0.9289079904556274, 0.4281047582626343, 0.5519875884056091, 2.114736795425415, 1.9581376314163208, -1.2140480279922485, 0.4933476448059082, -0.526025116443634, 1.7170816659927368, 1.5962812900543213, 2.764193296432495, 1.196428894996643, -0.04209812358021736, -0.03005354106426239, 0.6943859457969666, 2.3208658695220947, -1.1006474494934082, -0.4544893205165863, 0.5611380934715271, 1.1657969951629639, 0.2285061776638031, 0.7078961133956909, -0.44308990240097046, -0.3570430874824524, 0.5144039392471313, -1.7453961372375488, -0.338876336812973, -1.4441355466842651, 2.232175350189209, 1.2395421266555786, 1.448668122291565, -1.744889259338379, 0.7764650583267212, 0.7187661528587341, -1.453138828277588, 1.3120050430297852, -1.346238374710083, -0.3758120834827423, -0.5736891627311707, -0.0980272889137268, 0.188460573554039, -0.4520905315876007, -0.20313654839992523, -0.8700850009918213, -1.1270664930343628, -1.2874642610549927, 0.005196304060518742, 0.8012472987174988, 0.048740942031145096, 2.012164831161499, 0.06897429376840591, 2.0151326656341553, -0.0034922233317047358, -1.3947021961212158, -1.241377830505371, -1.1172159910202026, 0.0301017127931118, 1.211356520652771, 0.7748884558677673, -0.4304250180721283, -1.8232005834579468, -0.2815828323364258, -2.65708065032959, -0.16625791788101196, 0.6231661438941956, -0.4243551790714264, 1.1988664865493774, 0.27188965678215027, -1.0822292566299438, 0.2552679777145386, -0.13981346786022186, -0.32388240098953247, 1.048309564590454, -0.5194138288497925, 0.3614470362663269, 0.1339821070432663, 0.5259539484977722, 1.145473837852478, 0.3522428870201111, -0.4076310992240906, 0.36759334802627563, -1.017531156539917, 0.2148086130619049, -0.04889495670795441, 0.47029387950897217, -1.3807812929153442, 0.3029191493988037, 1.1622289419174194, -1.2841131687164307, 1.626227855682373, 1.6872401237487793, -0.827844500541687, -1.0802712440490723, -0.44171467423439026, 0.3015174865722656, 0.42699357867240906, -0.1937207281589508, 0.7223203182220459, 1.722683072090149, -0.7431226372718811, 0.40491876006126404, -0.46754613518714905, 0.43654975295066833, -0.17765292525291443, 0.1134209930896759, 0.7696703672409058, -1.2001395225524902, 0.9032711982727051, 1.0296640396118164, -0.3584772050380707, 2.269404411315918, 0.05116423964500427, 0.3559281527996063, -0.5536041855812073, -1.0229036808013916, 1.1963682174682617, -0.04227820038795471, 1.6909884214401245, 0.690850555896759, -0.7797942757606506, -0.19872692227363586, -0.7687655091285706, 0.4093112051486969, -2.0408153533935547, 0.3102903366088867, 1.5190284252166748, -0.38903525471687317, 1.7774417400360107, -2.876171827316284, -1.4237453937530518, 0.33517804741859436, 0.6292291283607483, 0.3699019253253937, 0.06470424681901932, 0.5941779017448425, -0.7947240471839905, 0.6457682847976685, 0.006451009307056665, -0.6462814807891846, -0.3783028721809387, -0.19002604484558105, 0.9077231287956238, -1.5744576454162598, 0.026039553806185722, 2.4004108905792236, -1.2445175647735596, 1.1751995086669922, 1.0838826894760132, 0.6991512775421143, 0.8859894275665283, -0.17688776552677155, 0.7341009378433228, 0.31323498487472534, -0.8674446940422058, 0.15807613730430603, 0.7771964073181152, 1.5392661094665527, -0.9501602053642273, -0.07171259075403214, -0.7901288866996765, 1.5322474241256714, 0.785179853439331, 0.2492833137512207, -0.010857854969799519, -0.795641303062439, -0.09180858731269836, 0.8557391166687012, -0.6856364011764526, 1.4079642295837402, -0.5985476970672607, 0.248438760638237, -1.2593867778778076, -0.553802490234375, 2.3395678997039795, 0.022504808381199837, 0.06561349332332611, -1.143758773803711, -0.017412343993782997, 0.31883788108825684, 1.4384562969207764, -0.8526768088340759, 0.3191969692707062, -0.7644548416137695, 1.524192214012146, -0.028109487146139145, -0.839331328868866, 0.39860987663269043, -1.2962009906768799, -0.02605830505490303, 1.5492045879364014, -0.20799386501312256, -0.09406009316444397, 0.33813783526420593, -0.8211093544960022, 1.9764325618743896, 0.5629050135612488, -0.32756322622299194, 1.6913517713546753, 0.6532818078994751, -0.31176379323005676, 0.44938284158706665, -1.259702205657959, 0.40759041905403137, 1.6025749444961548, -1.859961986541748, -0.5539885759353638, 0.85600346326828, -0.05300150439143181, 0.07263153791427612, -0.522156298160553, 1.3275033235549927, 1.4374887943267822, -0.4170385003089905, -0.3591255843639374, 0.25680187344551086, -0.013351356610655785, 1.4554063081741333, -0.13549545407295227, -1.3688045740127563, -0.40551722049713135, 1.1280821561813354, 0.5029172301292419, -1.0611029863357544, -1.0104522705078125, 0.14294296503067017, 0.5879071354866028, 0.0021210662089288235, -0.9708309769630432, -0.19295425713062286, 0.5515238046646118, 0.31765124201774597, 1.2556043863296509, -1.0475901365280151, -0.5526689291000366, -0.4280662536621094, 1.6944420337677002, -1.769653558731079, -0.6486740112304688, -0.5877128839492798, -1.464113712310791, -1.2933331727981567, 1.2807316780090332, -0.5280139446258545, -0.4246220588684082, 0.7458749413490295, -0.3079804480075836, -0.6183878183364868, 0.7552774548530579, 0.5542691946029663, -1.4336105585098267, -0.6382102370262146, 0.15514832735061646, 1.0108188390731812, 0.5364523530006409, -0.44424429535865784, -0.38110587000846863, -0.6428369879722595, 0.27403172850608826, 0.6406567692756653, -0.20080843567848206, -0.2526605427265167, 3.0327887535095215, 0.2501639425754547, -1.1299604177474976, -0.8442785739898682, 1.0372523069381714, 0.44162681698799133, -0.6419585943222046, 0.8285185694694519, 0.5546774864196777, 0.09084887057542801, 0.6294361352920532, -0.17280429601669312, -0.09042131900787354, -0.5124975442886353, -0.6625651121139526, -0.3136107921600342, -0.08013904839754105, -0.33317965269088745, 0.5961011052131653, 0.051541294902563095, 0.531052827835083, -1.2610938549041748, -0.1492239087820053, 2.6218504905700684, -1.8993422985076904, -0.39905020594596863, 0.8905107378959656, -1.0748168230056763, -0.3267831802368164, -0.5191866159439087, 1.0979162454605103, 0.6179390549659729, -0.1545988768339157, 0.16528688371181488, 0.9194448590278625, 0.18609057366847992, 1.2245789766311646, 0.26450565457344055, 1.2876938581466675, -0.212272509932518, -3.1973252296447754, -0.7422914505004883, -0.32687655091285706, 0.8246963024139404, 1.2828248739242554, 0.27556294202804565, -1.1257598400115967, 0.6726349592208862, -0.355067640542984, 1.4451042413711548, 0.2678565979003906, 1.3125988245010376, -0.5538498163223267, 1.0503129959106445, -0.1328173726797104, 0.07275740057229996, 0.32907915115356445, -1.7115274667739868, -0.26569998264312744, 0.3073482811450958, -0.39830270409584045, 0.7392411828041077, 1.0029979944229126, -2.0958940982818604, 0.04209976643323898, -0.7154191732406616, 1.724757432937622, 1.2389862537384033, 0.5503318309783936, 0.7972920536994934, -1.6720092296600342, -0.6528204679489136, 0.7636919617652893, -0.15273696184158325, 1.061703085899353, 0.0196399986743927, -0.7000904679298401, 0.004660062957555056, 0.28962382674217224, -0.27102038264274597, 0.34113961458206177, -1.8897912502288818, -0.948765754699707, -0.605316698551178, 0.8784319162368774, 0.2693556249141693, -0.1470542550086975, -0.9897845387458801, 0.7386006116867065, -0.22168049216270447, -3.368130922317505, -2.3096022605895996, -0.03406863659620285, -0.9158508777618408, -0.4598029553890228, 0.35224345326423645, 0.29681921005249023, 1.011531114578247, -0.32251742482185364, -0.4051606059074402, 0.3178749680519104, -0.8715466856956482, -0.0414125882089138, 1.522215485572815, 0.010471935383975506, -1.0496472120285034, 0.09147096425294876, 0.11820223927497864, 0.47401630878448486, -0.12786810100078583, -1.48472261428833, 0.8868207335472107, -0.03940741717815399, -0.940779447555542, 1.6393126249313354, -1.216558814048767, -1.2944273948669434, -1.0340200662612915, 0.3539007306098938, -1.1471648216247559, -1.0994278192520142, -0.0036350174341350794, -0.615232527256012, 0.3282400071620941, 0.925593912601471, 0.8444961905479431, 1.1797758340835571, 2.1588246822357178, -0.1407792866230011, -1.7280898094177246, -0.8170121312141418, -0.5234746932983398, 0.5971304178237915, -1.3058894872665405, 2.3397483825683594, -1.0664284229278564, -1.6854186058044434, 0.9121590852737427, -0.11699618399143219, 0.022282280027866364, -0.6826263666152954, 1.004921793937683, -0.4774576723575592, -0.7689346075057983, -0.7117135524749756, -0.8575946688652039, -0.11123000085353851, -0.7382857799530029, 0.0694325864315033, 0.8980101346969604, -0.9656922221183777, 0.27779972553253174, 0.5080257058143616, -0.5542606711387634, 0.3584436774253845, 2.336575984954834, 1.0684970617294312, -1.4750134944915771, 2.095557689666748, -1.1117265224456787, -2.167515993118286, -0.5717896819114685, 0.15720608830451965, 0.7137628793716431, 0.5099607110023499, -0.1372496485710144, -1.59423828125, 1.3655394315719604, 1.3843674659729004, -1.9415994882583618, -0.1771923452615738, 0.07989860326051712, 0.4207659959793091, 0.23510822653770447, -0.3669532537460327, -0.20255769789218903, 2.624791383743286, 0.7132858037948608, 1.6438442468643188, 0.14789916574954987, -0.830193817615509, -0.7203181385993958, 0.26106080412864685, 0.047839365899562836, -0.872374951839447, -0.6689184308052063, 1.010474681854248, -0.22761447727680206, 0.15563362836837769, 1.5775543451309204, 0.5946617722511292, 0.8252483606338501, -1.481609582901001, 1.3676409721374512, -1.4084261655807495, -1.3196879625320435, -1.576115369796753, 0.21771638095378876, 0.7046127915382385, 0.33038464188575745, -0.22345373034477234, 1.0641086101531982, -0.36737021803855896, 0.3390917479991913, -1.5396257638931274, -0.30438756942749023, 1.5053037405014038, -0.5094513893127441, -1.1571893692016602, -0.017186734825372696, -0.8259425759315491, 0.36940521001815796, -1.405798316001892, -1.2677677869796753, -0.8572482466697693, 0.5887587070465088, -0.4934147596359253, -0.690229058265686, -1.2189470529556274, 0.4190245568752289, 0.10732274502515793, -2.092034339904785, 0.10301105678081512, -0.41976210474967957, 0.9351130127906799, 0.022477183490991592, -0.8627476692199707, -0.30530625581741333, -1.3111417293548584, -0.1013786792755127, 0.5238258242607117, 0.6543739438056946, 2.433657169342041, 1.215023159980774, -0.6615378856658936, 1.1396610736846924, -1.1402199268341064, 0.7706942558288574, -0.9674401879310608, -0.9516942501068115, 1.6063146591186523, 0.7142020463943481, -0.5348678231239319, 0.5894747376441956, 0.4467388987541199, -0.07112562656402588, -0.05305573716759682, 0.8938539624214172, 2.2307958602905273, -0.3790305554866791, 0.15104328095912933, 0.627585232257843, -0.12684552371501923, -0.17817024886608124, -0.3834955096244812, 1.536689281463623, 0.3814392685890198, 1.7902413606643677, -2.7162036895751953, -1.0978103876113892, -0.8878192901611328, 0.5184484720230103, 0.14691506326198578, -0.4862208366394043, 0.9954713582992554, 0.35025811195373535, -0.8137960433959961, 0.5330418348312378, 1.7943512201309204, -0.0663096010684967, -0.04335387051105499, -0.8935660123825073, 1.0249900817871094, -0.06322775036096573, 0.26867902278900146, -0.21017850935459137, 1.289990782737732, -0.5457123517990112, -1.336874008178711, 0.8918139338493347, 1.246456265449524, -1.1165462732315063, -0.3501158356666565, -0.4576830565929413, -0.5768381357192993], [0.14965549111366272, -1.491554617881775, -1.0637809038162231, 0.35360804200172424, -3.17920184135437, 1.563592553138733, 1.1564013957977295, -1.7125197649002075, 0.07985151559114456, -0.6817618608474731, -0.029025360941886902, 0.3520474135875702, -0.2848875820636749, 1.1515097618103027, -1.8092175722122192, 1.249282956123352, -0.9005767107009888, -2.008284330368042, 1.3214471340179443, -2.15108060836792, -0.43641820549964905, -1.482311725616455, 1.073400855064392, 1.2974668741226196, -0.6855171322822571, 0.08503289520740509, 0.012683933600783348, -1.0316909551620483, 0.8174421787261963, -0.4592568576335907, -0.7298163771629333, -0.8794745802879333, 1.225812554359436, 0.9698207974433899, 0.24781695008277893, 0.6994137763977051, 1.459619164466858, 0.6024172306060791, -1.617679238319397, -0.5143892765045166, 0.40566015243530273, 0.4425233006477356, -0.5481168031692505, 0.07472959160804749, -1.5763412714004517, 0.6991446018218994, 1.9092061519622803, -0.13290667533874512, 0.21505089104175568, 0.8672307729721069, -0.460519015789032, -0.23364044725894928, 1.6198375225067139, -0.06645451486110687, 0.8493260741233826, 0.9793847799301147, -1.0262953042984009, 0.2763028144836426, -0.0005938977701589465, 0.07346165180206299, -1.342557430267334, -0.7557534575462341, -0.4281792938709259, -0.42511963844299316, -1.7312575578689575, -1.2681280374526978, -1.5636810064315796, -0.20840668678283691, 0.25846248865127563, -0.6211945414543152, -0.44068875908851624, 0.8776659369468689, 0.14500385522842407, 0.6768473386764526, 0.8477104902267456, 0.042929328978061676, 0.41515833139419556, 1.7537548542022705, -0.19541364908218384, -0.018868977203965187, 1.1995751857757568, -1.0662156343460083, 1.1031633615493774, 0.4703371822834015, 0.7591918706893921, 0.6354151368141174, 0.6821281313896179, 1.2174034118652344, 0.7436591982841492, -1.3464863300323486, 0.7454426884651184, -0.5722411870956421, -0.1784428507089615, 0.7802395820617676, -0.10311871767044067, 1.7498664855957031, 1.518655776977539, -1.1023480892181396, 0.17991025745868683, 0.8716568350791931, -0.79693204164505, -0.01645027846097946, -0.8161193132400513, -1.3872417211532593, 0.25030848383903503, -1.2770308256149292, 0.03472026810050011, -1.2048404216766357, 0.06921268999576569, -0.9525904059410095, -0.14363671839237213, -0.035154182463884354, -0.03679468110203743, 1.9879608154296875, -0.8089779019355774, -0.9629277586936951, 1.2761669158935547, 2.2754106521606445, -0.35258930921554565, -0.05201847478747368, -0.3350284993648529, 0.43343499302864075, 0.6602402925491333, -1.182992935180664, 0.4987231194972992, 1.5480815172195435, -1.0156586170196533, -0.3556382954120636, -0.7237752676010132, 1.8070052862167358, 0.47004565596580505, 1.2591570615768433, -0.027568338438868523, 0.5916357636451721, -1.5113760232925415, 0.6123908758163452, -1.4577571153640747, -0.6121761202812195, -0.4140898883342743, -0.7472947835922241, 0.5008535981178284, 1.6648616790771484, -0.4480116069316864, -1.4730173349380493, -1.0311616659164429, -2.2309718132019043, -0.34046217799186707, -0.8007585406303406, 1.1382006406784058, 0.40424075722694397, -1.527704119682312, 0.673906683921814, -0.9290831089019775, -0.6876210570335388, 0.11462192237377167, -0.18656522035598755, -2.258117198944092, 1.1220331192016602, 1.5038813352584839, 0.3857108950614929, -0.7754122018814087, 0.9304159879684448, -0.09212690591812134, -1.5141453742980957, 0.6374602913856506, 0.18965762853622437, 0.057834651321172714, 1.2732782363891602, 0.26877570152282715, -1.5540863275527954, 1.4321296215057373, -1.4730525016784668, 0.8646093010902405, 0.1255064457654953, 0.9189987778663635, -0.7867128849029541, 0.5738915205001831, 0.6558281779289246, -2.3711752891540527, -0.2445572167634964, 0.9095531702041626, -0.9401888251304626, 0.7817405462265015, 0.3393794596195221, -0.40661126375198364, 0.21878035366535187, -1.4972115755081177, 0.3343912363052368, -0.47313836216926575, -1.0648903846740723, -0.878998339176178, 0.3201026916503906, 0.48558616638183594, -0.5840558409690857, -0.34344708919525146, 1.5754927396774292, -0.3029629588127136, -0.4605550765991211, 1.2257755994796753, -0.31105899810791016, -0.26347923278808594, 1.0936180353164673, 1.5625826120376587, 1.8506686687469482, -0.7264941930770874, -0.9232779741287231, -0.23259811103343964, -1.6785335540771484, 0.9527415037155151, 0.7904755473136902, 0.7134471535682678, 0.4699004590511322, -1.8159209489822388, 0.4173510670661926, -0.9864816069602966, -0.858177900314331, 0.7475154399871826, -0.3600969910621643, -0.3793846070766449, 1.4501726627349854, 1.6127197742462158, 0.7937405705451965, 0.233157679438591, 0.014131467789411545, -0.22087816894054413, 1.204376459121704, -0.007001341320574284, -0.27641528844833374, 1.0565152168273926, 0.5614246726036072, -0.3590587079524994, -2.5655839443206787, 1.308051586151123, -0.8304069638252258, -0.424307256937027, -0.015118585899472237, -1.8960460424423218, -0.8202272653579712, -0.8749810457229614, -0.18119554221630096, -0.48284080624580383, -2.354930877685547, -1.3301475048065186, -0.4438696503639221, -0.16629445552825928, -0.6979891657829285, -0.3636491894721985, 0.7149167656898499, 0.5126144886016846, -0.8979439735412598, -0.19870862364768982, 0.05200659856200218, -0.6420115232467651, 0.058618828654289246, 0.7145540118217468, 1.6439800262451172, -0.22756493091583252, 0.040347758680582047, 1.1609944105148315, 2.41642689704895, -0.33786460757255554, 0.8018683791160583, -1.7254292964935303, -1.1091876029968262, 0.12698277831077576, 1.3135021924972534, -1.531346321105957, 0.8173348307609558, 0.3333097994327545, -1.4542272090911865, -0.47633621096611023, -1.502066969871521, 1.1614056825637817, -0.31103554368019104, 0.3094598650932312, -0.08579867333173752, -1.28972327709198, 0.36288148164749146, -0.4913143813610077, -0.5731586813926697, -0.016042280942201614, -0.6272801160812378, -0.37086668610572815, -1.1790655851364136, -0.035284508019685745, 0.05596764385700226, -0.4844990372657776, 0.33613163232803345, 1.725181221961975, -1.6896333694458008, -1.2531510591506958, -0.2500617802143097, 1.2643260955810547, -0.04697944596409798, 0.14517901837825775, -0.7423775792121887, -0.07918553054332733, 0.14631982147693634, -0.6970233917236328, -0.41956520080566406, 1.7310080528259277, 0.0817437693476677, 1.3733322620391846, 1.7691127061843872, 1.3384782075881958, -0.40388041734695435, 1.1355067491531372, 0.6316750049591064, -0.866427481174469, 0.14317838847637177, -0.8769381642341614, -0.4754085838794708, -1.1572238206863403, -0.3784083127975464, -0.05112013220787048, 0.9769417643547058, -0.11754637956619263, 0.1387311816215515, 0.5318052172660828, -0.9157505631446838, 0.8942441940307617, -0.9708136916160583, -0.6455358862876892, -0.24904842674732208, 0.8791006207466125, 1.8389564752578735, -1.5303019285202026, -0.508893609046936, 1.3915283679962158, -1.1184805631637573, 0.6117892265319824, -1.1416670083999634, 1.8796217441558838, -0.5276316404342651, -0.01496800035238266, 1.8625125885009766, -1.0284500122070312, 0.48256710171699524, -1.222648024559021, -0.9529077410697937, 0.2209671586751938, -0.16072900593280792, 0.1610775887966156, -0.05677657946944237, -0.1790522038936615, -0.687584638595581, 0.19507569074630737, 0.21835923194885254, -1.3492194414138794, -0.760520875453949, -0.2895846366882324, 0.35947075486183167, 1.0768702030181885, 0.10900893062353134, -1.1587421894073486, 0.36132556200027466, -0.0936257541179657, 0.01944953016936779, 0.008099354803562164, -0.7096765637397766, -0.7931474447250366, 1.7204372882843018, -1.3007104396820068, 0.9274441003799438, -0.45635703206062317, 0.6793196201324463, 1.0789666175842285, 0.47967585921287537, 0.6615141034126282, -1.3784664869308472, -1.6783547401428223, -1.195677638053894, -1.6964483261108398, -0.41230958700180054, 1.2834224700927734, 0.21854014694690704, -1.1469181776046753, -1.9696276187896729, -0.4319196343421936, 1.678307294845581, 0.43167272210121155, 0.6849456429481506, 0.9013407230377197, 0.8702536821365356, 1.3015525341033936, -0.24024698138237, 0.34100082516670227, 0.22730860114097595, 0.03618234023451805, 0.8221330642700195, -1.6582733392715454, 1.9524165391921997, 0.09963423758745193, -0.37975263595581055, 0.5226337909698486, -0.2206874042749405, -1.1917407512664795, 0.6562316417694092, 0.3462374806404114, -1.1014138460159302, -0.667894721031189, 0.4117473363876343, -0.0583142526447773, -0.9842610359191895, -2.295559883117676, -2.272953510284424, -0.607743501663208, 1.1565309762954712, -0.5875114798545837, 1.0259392261505127, 0.5634885430335999, 0.41558071970939636, -1.650762677192688, -1.667183756828308, 1.4417836666107178, -1.6808714866638184, 0.21280762553215027, 0.26793423295021057, 1.7452479600906372, 0.20757897198200226, 1.726163387298584, -1.3431987762451172, 0.021999523043632507, 1.4964497089385986, 0.8092851042747498, 0.055044952780008316, 1.6542953252792358, -1.1650285720825195, 0.5896961092948914, 1.5536143779754639, 0.9963958263397217, -1.2183208465576172, -1.7568480968475342, 0.8060029149055481, -1.2054160833358765, 0.7963414788246155, -0.43713048100471497, 0.2521674633026123, -0.8750877380371094, -0.4563371539115906, 0.9716715812683105, -0.5551797151565552, -0.9026677012443542, 0.09948752820491791, -1.729548692703247, 0.31251445412635803, -1.2909263372421265, -1.7986056804656982, 1.291643738746643, -0.12789475917816162, -0.7598313689231873, -1.2095297574996948, 0.8467562794685364, -0.0962551161646843, 0.7706179618835449, 0.2581917643547058, 0.6910942196846008, 0.459950715303421, -0.8686962127685547, 0.6061665415763855, -0.1940588802099228, 1.2650067806243896, -1.4739444255828857, 0.930746853351593, 0.7644349932670593, 0.6624082326889038, -0.5822082161903381, -0.7854078412055969, -2.491895914077759, -0.7469161748886108, -1.8885626792907715, 0.17062757909297943, -1.474095106124878, 0.08489080518484116, 0.6631515026092529, -1.2112841606140137, 0.2779313623905182, -0.7476906776428223, -0.40755629539489746, -0.6230360269546509, -0.8852657079696655, -0.4430862367153168, 0.32095080614089966, 1.7993611097335815, -0.7693184018135071, 1.105573296546936, -1.990950107574463, -0.27062293887138367, 1.0027316808700562, 1.2764590978622437, 0.12790800631046295, -1.797019600868225, -0.5881077647209167, -0.06377599388360977, -0.9952208995819092, -1.1824772357940674, -0.440727561712265, -1.032343864440918, 0.8344742059707642, -1.0445982217788696, 1.2606213092803955, 1.649781346321106, -0.2660657465457916, 0.16806940734386444, 1.1371486186981201, -0.33216676115989685, 0.16026078164577484, -0.38953179121017456, -0.43997788429260254, 0.20457251369953156, -0.032535016536712646, -1.0894495248794556, -2.0167384147644043, 0.3889247477054596, -0.19766610860824585, 1.6571803092956543, -0.8549721240997314, -0.6275039315223694, 0.5878472328186035, -0.26557230949401855, 0.9952871203422546, -1.968658447265625, 1.9680484533309937, -1.1774553060531616, 0.7289393544197083, 0.9758036136627197, 1.1675806045532227, -0.14688406884670258, 0.7574070692062378, 0.44427213072776794, 0.999772310256958, 0.267396479845047, -0.272927850484848, -0.9574563503265381, 0.44503501057624817, -1.904536485671997, 1.890998125076294, 0.18786314129829407, -1.2992528676986694, -1.085532546043396, -1.303893804550171, 1.6496217250823975, 0.22786769270896912, 0.22172820568084717, -0.5423039793968201, -0.6790198683738708, 0.4157044589519501, -0.5752183198928833, -1.085133671760559, -1.291991114616394, -1.2087903022766113, 0.342540442943573, -2.7511041164398193, 0.21520927548408508, 0.3510141670703888, 0.5048810839653015, -0.6921280026435852, 0.8162018656730652, -1.625109314918518, -2.0279526710510254, -0.1343332827091217, -0.018703630194067955, -0.37448880076408386, 0.963275134563446, -1.1473734378814697, -1.4101110696792603, -0.6187125444412231, 1.462633490562439, 0.39838236570358276, -0.3432769775390625, 1.3980098962783813, 1.0878690481185913, -1.4455394744873047, 0.09088392555713654, 0.6813525557518005, -0.23468346893787384, -0.26415836811065674, 0.15788103640079498, -0.7700742483139038, 0.6172837018966675, -0.27719977498054504, 0.4938502311706543, 1.87418794631958, -0.21515637636184692, 0.9899240136146545, -0.9335843920707703, -0.6509433388710022, -0.5593779683113098, -0.08478440344333649, -0.8194022178649902, -2.529252052307129, 1.1868057250976562, -0.42354029417037964, 0.2609855830669403, -0.6624962091445923, -1.34349524974823, -1.1003546714782715, 0.9346004128456116, -1.3618415594100952, 0.014760278165340424, -0.7965556979179382, -0.2506099343299866, 1.2389417886734009, -0.9673931002616882, -1.5408862829208374, 1.0983920097351074, -0.22254888713359833, -1.088544487953186, -0.4444172978401184, -3.2767648696899414, -1.2107114791870117, 0.9286291003227234, 0.015334278345108032, 0.15434347093105316, -0.8819628953933716, 0.3236989974975586, 1.6680817604064941, -0.4011746644973755, -1.177287220954895, -1.5417760610580444, -0.9814926385879517, -0.3343605697154999, 1.357621669769287, 0.07683753222227097, 0.395333856344223, 1.1398229598999023, 0.034118831157684326, -0.28967809677124023, 1.4265586137771606, 1.1838690042495728, -1.0584831237792969, -1.09712553024292, -0.7846423387527466, -2.112257242202759, -0.48562148213386536, -1.0510122776031494, 0.25098440051078796, 0.9650667905807495, -0.3534175753593445, 0.988916277885437, 0.5771833658218384, 0.3939874470233917, 1.1162947416305542, 0.18399958312511444, 0.0500103235244751, -2.6600677967071533, -0.9808474183082581, -1.346091866493225, 0.6218720078468323, 0.576977014541626, -1.5769799947738647, 0.5531636476516724, 0.6698595881462097, -0.0978311151266098, -0.6754589080810547, 2.088322162628174, -0.33551427721977234, 1.1033071279525757, -1.1558327674865723, 1.458213448524475, 0.1366700679063797, -0.36307036876678467, 2.434892177581787, -2.705960273742676, 1.103027582168579, 0.7341600060462952, 0.5564842820167542, 0.9895196557044983, 0.5697103142738342, -0.009056228213012218, 0.6780434846878052, 0.7708534598350525, -1.2040032148361206, -0.6317722201347351, 1.3352221250534058, -1.0274262428283691, -1.7757527828216553, -0.3784850835800171, 0.8749857544898987, 0.8628458380699158, 0.571498453617096, 0.6628300547599792, -0.31853121519088745, -1.1830365657806396, 0.8177659511566162, -1.1481668949127197, -0.8159962892532349, 0.41888073086738586, -1.309776782989502, 0.40338075160980225, 1.4967496395111084, 0.25224602222442627, 0.8154075741767883, -0.09019719809293747, -0.24890580773353577, -2.2235589027404785, -0.6841825842857361, 1.6394329071044922, 1.2954552173614502, 0.2533673346042633, -0.6181570291519165, -1.7794885635375977, -0.9698805212974548, 1.2058510780334473, 0.6829310059547424, 0.3222096264362335, -0.1714586317539215, 0.5595437288284302, -1.528563141822815, 0.5708796381950378, -1.7857686281204224, 1.1136510372161865, -0.47198715806007385, -0.13776566088199615, -0.2045222669839859, 0.11832211911678314, -0.49608683586120605, -0.5376894474029541, 0.12471619993448257, 0.026286577805876732, 0.6298507452011108, 0.8551328778266907, 0.0045812660828232765, 1.398895263671875, 1.0191391706466675, 2.50693416595459, -1.7388826608657837, 1.2200802564620972, -0.6252899765968323, -1.3173046112060547, -1.0580735206604004, -2.045938014984131, -0.7364511489868164, 1.693942666053772, 0.2558346688747406, 0.29391270875930786, -1.7358828783035278, 0.031200258061289787, -1.451796293258667, 0.41172710061073303, 0.4006991982460022, -0.4762510657310486, -0.3379201591014862, 1.4400140047073364, 0.9592827558517456, 0.5916836857795715, -0.9151420593261719, -0.8090569972991943, -0.8549340963363647, -0.6330438852310181, 0.5371413230895996, 0.009343230165541172, 2.3371593952178955, -0.7746362090110779, -0.08671263605356216, -1.3003100156784058, -0.7109808921813965, -0.0989530086517334, -1.3163187503814697, 0.6698100566864014, 0.7380404472351074, -0.34099268913269043, 0.11985725909471512, 0.7342818379402161, 0.5900465250015259, 1.374537467956543, -0.6150607466697693, -1.518580436706543, -1.2541277408599854, -0.8308272957801819, 0.9464707970619202, -1.2391778230667114, -1.0659558773040771, 0.2095986306667328, 0.1250665783882141, 0.5904549956321716, -0.2756563425064087, 1.0586647987365723, 0.9614983201026917, -0.11606776714324951, -0.6752357482910156, 0.09138820320367813, -0.6380863785743713, 0.16612035036087036, 0.1914403885602951, -0.5562776923179626, -0.7759703993797302, -0.02737957425415516, -0.4352891743183136, -1.166088581085205, 0.42098215222358704, 1.965022325515747, 0.4198386073112488, -0.47941914200782776, 0.0040503451600670815, -0.3487560451030731, 0.19807584583759308, 0.33686593174934387, 1.984931468963623, -0.5051001310348511, -0.44135037064552307, -0.062217846512794495, 1.2040073871612549, -0.8786677718162537, -0.4774189591407776, -1.7650854587554932, 0.1967954784631729, -0.5115733742713928, 0.9744593501091003, -0.25159090757369995, 1.261156439781189, -1.0834842920303345, 0.5557536482810974, -0.12160810083150864, 0.40189406275749207, 1.3859264850616455, 0.6651268601417542, -2.067812919616699, -0.6406016945838928, -0.9493097066879272, 1.8824619054794312, 3.280153751373291, -0.24540975689888, -0.5265005230903625, -0.2141183763742447, 0.38020163774490356, 0.34177693724632263, -0.019589979201555252, -1.1971886157989502, -0.3767266273498535, 0.049382105469703674, 0.39434680342674255, 0.5256752967834473, -1.3001480102539062, -1.2895770072937012, -1.8737304210662842, 2.800427198410034, 0.755333423614502, 1.2993855476379395, -0.8467835783958435, -0.09057050198316574, 0.29821619391441345, -0.41660815477371216, -0.27166590094566345, -0.811107337474823, 0.26127469539642334, 1.1405514478683472, -1.758788824081421, -0.807300329208374, -1.7115775346755981, -0.4971142113208771, 0.3416931629180908, -1.578899621963501, 0.9494296908378601, 0.316119521856308, -0.10147725045681, 0.2421659678220749, 0.4806317090988159, -0.4402586817741394, 0.6173098087310791, 0.26205340027809143, 1.1760112047195435, 0.09365624934434891, -0.8711603283882141, 0.1307775378227234, -0.5875937342643738, -1.0389385223388672, 0.2329409420490265, 2.195748805999756, -1.5830273628234863, -0.6707608699798584, -1.0728152990341187, -0.8678363561630249, 0.15733304619789124, 0.4203086495399475, 1.0961352586746216, 0.46744969487190247, 0.40712395310401917, 0.40536409616470337, -0.9165846705436707, 0.2221750020980835, 0.4941696226596832, 1.680627465248108, -0.3054191768169403, 0.0069199842400848866, -1.293444037437439, -2.2592194080352783, 0.5787597298622131, -1.8691554069519043, -0.46071311831474304, 0.16504941880702972, 0.950607180595398, -1.7244293689727783, -1.775563359260559, -0.8693829774856567, -0.6161679029464722, 0.8239437341690063, -0.1442456841468811, 0.26563647389411926, 0.23678898811340332, 0.6567927002906799, 1.886731743812561, -1.2890214920043945, -0.15700192749500275, -0.6683670878410339, 0.8128973841667175, 1.8371645212173462, 0.05784802883863449, 0.7660938501358032, 0.07516265660524368, -2.1146795749664307, 0.770420253276825, -0.6936911344528198, -0.1366749107837677, 0.7222565412521362, 0.14995619654655457, -1.2728915214538574, -0.051298193633556366, 0.5952314138412476, -0.49941086769104004, -0.38246262073516846, 0.20924504101276398, 0.23022787272930145, -1.9349372386932373, 0.4143593907356262, 0.4277094900608063, 2.043739080429077, 0.36386874318122864, -0.4831792414188385, -0.5058428049087524, -1.162527084350586, -0.13815711438655853, 0.0018122484907507896, -0.8197781443595886, 1.5842387676239014, 0.061642203480005264, 1.1044731140136719, 0.07772275060415268, 0.46590903401374817, 1.5197302103042603, -0.7588472366333008, -0.24394887685775757, -0.0152298454195261, -0.25729089975357056, 1.0591479539871216, 0.09215617924928665, -0.8008131980895996, -0.27781200408935547, -0.9327519536018372, -0.07889856398105621, 3.010760545730591, 2.399097204208374, 0.47408387064933777, -2.834630250930786, 0.34806984663009644, -1.8979653120040894, -1.2398585081100464, 1.044586181640625, 0.3533288240432739, 1.089694857597351, 0.7029647827148438, -0.07842410355806351, 0.03744549676775932, 0.20727413892745972, 0.2821648418903351, -0.6410698890686035, 0.6117787957191467, -0.11400506645441055, 1.1450978517532349, 1.0011290311813354, -0.742594301700592, 0.9989617466926575, -0.8923938274383545, 0.3893827795982361, -0.9659752249717712, 0.7761930823326111, -0.26769575476646423, 0.7900490760803223, -0.5749863386154175, -0.002277714665979147, -0.7217811942100525, 1.1056474447250366, 1.3591296672821045, 0.340760201215744, -2.003042697906494, -0.99223393201828, -2.0731356143951416, 0.6813092231750488, 1.9603759050369263, -0.20095454156398773, -0.44533398747444153, -0.6717668771743774, -0.7437477111816406, 0.8335294127464294, -0.9974727034568787, 0.8636908531188965, 0.8833499550819397, -1.1750718355178833, 1.563979148864746, -0.12831705808639526, 2.354591131210327, 1.0346758365631104, 0.04031844809651375, 0.4352925419807434, 0.6727083325386047, -0.7938040494918823, 1.0156350135803223, 2.457658529281616, -0.4799310863018036, -1.4346975088119507, -2.1659440994262695, 0.5750409960746765, 1.3179199695587158, 0.19392843544483185, 0.19616958498954773, -1.4442867040634155, -1.3752202987670898, -0.9927902817726135, 0.2630034387111664, -1.0081007480621338, -0.9343637228012085, 0.590454638004303, -0.4495653212070465, -0.2859434485435486], [0.6918956637382507, -1.1185966730117798, 1.8570728302001953, -0.9562148451805115, 0.16659891605377197, -0.034661244601011276, -0.4604302942752838, -1.3978874683380127, 0.5205302238464355, -1.418480396270752, -1.5385808944702148, -0.5411959290504456, -1.0903899669647217, 0.20652417838573456, 2.27150559425354, 1.1312311887741089, -0.3847135305404663, 0.34596705436706543, -0.42793941497802734, 1.0477663278579712, -0.2741529047489166, 0.2275431901216507, 0.10897064954042435, 1.3709660768508911, 0.24018020927906036, 2.7302968502044678, 0.814379096031189, -0.05546741187572479, -0.646325409412384, -0.9425129294395447, -3.7209136486053467, -0.690019428730011, 0.14376546442508698, 0.45648616552352905, 0.4666755795478821, 3.0773098468780518, -0.18990692496299744, -1.3153671026229858, 1.3436267375946045, 0.8856486678123474, -0.1964452862739563, 1.893718957901001, 1.318166971206665, 0.7197914123535156, -0.9594730734825134, -0.022790104150772095, 0.19270940124988556, -0.11489551514387131, 0.4616985023021698, -0.47334858775138855, -0.13367263972759247, -0.38209301233291626, -0.944605827331543, 1.8455859422683716, -0.6093994379043579, 1.5703610181808472, 0.4148366153240204, -0.30066239833831787, -0.9105507731437683, -0.9183221459388733, 0.7509260177612305, 1.1076843738555908, -0.6131103038787842, 0.14586050808429718, 0.84801185131073, -0.1593094766139984, -0.3736110329627991, 1.1664060354232788, 0.10450785607099533, 0.776275634765625, -0.6304193139076233, -0.08580684661865234, -0.4407714307308197, -0.05307590961456299, -0.6443040370941162, -0.2751011848449707, -0.35419729351997375, 0.01857093907892704, 0.3986112177371979, 0.7975612282752991, -0.25896164774894714, -1.0239218473434448, 0.2829190790653229, 0.7807266712188721, -1.2807931900024414, 1.899229645729065, -1.4482921361923218, 0.6520751714706421, 0.367139607667923, 0.34288710355758667, -0.11562589555978775, 0.38127511739730835, -1.3503605127334595, 1.7020657062530518, -1.4248920679092407, -1.3285448551177979, 0.3346429169178009, -0.7016775012016296, -0.022432629019021988, -0.18522854149341583, -0.6925839781761169, -0.711478590965271, 0.24061402678489685, -0.4101293981075287, 1.0565603971481323, 0.31081488728523254, 0.9805168509483337, 0.37834396958351135, 1.554930567741394, -0.3294251263141632, -0.05681237205862999, 0.5130428075790405, 0.8143424987792969, 0.9424862861633301, 0.3253939151763916, -0.025207102298736572, 0.9746089577674866, -0.19079825282096863, -0.22555077075958252, 0.21277880668640137, -0.21606306731700897, -0.05223806947469711, -0.3323192000389099, -0.29457932710647583, 1.1549217700958252, 0.6426060795783997, 1.0103033781051636, -1.1807739734649658, 0.7738218307495117, -0.31348055601119995, -1.5999175310134888, -1.0148106813430786, 0.390864759683609, -0.03709203377366066, 1.7505500316619873, 1.1575788259506226, 1.1017065048217773, 1.7340471744537354, 1.1534690856933594, -0.03192133083939552, 1.4229238033294678, -0.18940185010433197, 0.31111249327659607, -0.5268028974533081, 1.0749540328979492, -1.599582552909851, 0.21640264987945557, -0.7441652417182922, -1.674434781074524, -1.033320426940918, 0.5165097117424011, -0.0397174172103405, 0.9170386791229248, -0.8433160781860352, 0.8012605309486389, 1.4242812395095825, 1.6295119524002075, -0.8302587866783142, -0.5067889094352722, -0.08021237701177597, 0.7189309597015381, 0.7933966517448425, -0.8572492599487305, 2.650510311126709, -1.5351908206939697, -0.6150496006011963, -0.2918044626712799, -0.07860937714576721, -1.09053635597229, 1.1695432662963867, -1.3431321382522583, 0.5237057209014893, 0.6928728222846985, -1.4786145687103271, 0.07158100605010986, 0.027585923671722412, -2.079909324645996, 0.14663641154766083, -0.1338166892528534, 0.18149887025356293, 0.21866914629936218, -0.0804186537861824, -1.3189613819122314, 0.8929449915885925, 0.48412850499153137, -1.340779423713684, -1.796572208404541, 0.9594893455505371, -0.1504027396440506, -1.4550220966339111, 2.4182727336883545, -2.490849018096924, -0.11104024201631546, -0.1333562284708023, -1.8099377155303955, 0.923105776309967, -0.9341732263565063, -1.7022181749343872, 1.6029634475708008, 0.4668138921260834, -0.3977409601211548, 0.0750749260187149, -0.1135784238576889, -1.5934122800827026, 0.8108488917350769, 0.21491041779518127, 0.9245507121086121, 0.49507492780685425, 0.2716217637062073, 1.279933214187622, 0.17826437950134277, -0.37684646248817444, 0.7516094446182251, -0.9180302619934082, 0.17631320655345917, 0.25599920749664307, 0.13290974497795105, 0.8479805588722229, 0.24141621589660645, -0.07331515848636627, 0.4180164635181427, 0.8862084746360779, 0.8711199164390564, -0.9278742671012878, -0.16193339228630066, -0.16287030279636383, -0.7143611311912537, -0.3801218569278717, -0.5332737565040588, 0.029744578525424004, 1.7939027547836304, -0.7850814461708069, 0.19960205256938934, 0.06607452034950256, 0.823525071144104, -0.0350462831556797, 0.25871288776397705, 0.3860889971256256, -1.5293028354644775, -1.5069043636322021, -1.3593826293945312, 0.42645469307899475, 0.9222305417060852, 0.6191962957382202, -1.4543958902359009, 0.6034418344497681, 1.7888487577438354, -1.2237170934677124, -0.05762508884072304, 0.5148962140083313, 0.6140950322151184, -0.06627070158720016, 0.22965650260448456, -1.104703664779663, -1.7752808332443237, 0.8603829741477966, 0.25240927934646606, -1.0492745637893677, 0.5350017547607422, 0.2024393230676651, -0.6469680070877075, 0.383027046918869, 0.8987182974815369, -1.738860845565796, -0.389148086309433, -0.05320197716355324, -0.9662345051765442, 0.2189973145723343, -2.4833576679229736, 1.5725537538528442, -0.013105357065796852, -1.070505142211914, -1.4924980401992798, 0.8051615357398987, 0.7688074707984924, -0.34712550044059753, 1.5510940551757812, 1.5378435850143433, -0.25521036982536316, 0.8820652365684509, 1.3152281045913696, -0.7743576765060425, -0.0971842110157013, -2.899230480194092, -1.6608474254608154, 2.728585720062256, -1.7112126350402832, 0.7307596206665039, -0.34122130274772644, 0.00152045045979321, -0.4174264669418335, 1.2451516389846802, -0.612304151058197, 0.22959119081497192, -0.3651355504989624, -0.6009863018989563, 0.48775357007980347, 1.4221490621566772, 0.14501482248306274, -0.17832371592521667, -1.134326696395874, 1.1389909982681274, -0.43686938285827637, -0.5814205408096313, 0.8049654960632324, 0.4783720374107361, 0.6027198433876038, 0.44394034147262573, 0.40479224920272827, 0.619785726070404, -0.9403008222579956, 0.06234622746706009, 0.8812689185142517, 0.3816307485103607, 0.3594057857990265, 1.7911360263824463, -1.5813367366790771, 0.4435337483882904, -1.1982064247131348, -1.0801193714141846, -0.9091824293136597, 1.2158948183059692, 0.19549240171909332, 1.7617746591567993, 0.41883277893066406, -0.3076050281524658, 2.226940155029297, -0.8976345658302307, -0.7374818325042725, 2.282996654510498, -0.3754839301109314, 0.275425523519516, -0.7859829664230347, 1.386473536491394, 0.10234308987855911, -1.6869044303894043, -0.8944603204727173, 0.559829592704773, -0.21721628308296204, 1.390006184577942, -0.8697205781936646, 0.18107551336288452, 0.5188522934913635, 1.3723942041397095, 0.22063954174518585, -1.3614306449890137, 0.1846785843372345, 1.4436099529266357, -0.6075192093849182, -1.488877296447754, 1.5086427927017212, -0.5900834798812866, -0.5407622456550598, -0.45081448554992676, -0.3844745457172394, 0.45958900451660156, 0.3778301477432251, 1.0147688388824463, -0.8588767647743225, 1.4809616804122925, 0.6782386898994446, 0.7919442057609558, -0.27111586928367615, -1.438103199005127, -0.09900643676519394, 0.8259325623512268, -0.03275440260767937, -0.06611382216215134, 0.34001481533050537, 0.5805348753929138, 0.3784385621547699, -0.8714473843574524, -0.8633149862289429, 1.5498400926589966, -1.01943039894104, 1.4661551713943481, -0.018626615405082703, -0.12878195941448212, -0.4967813789844513, 0.42148786783218384, -0.9604134559631348, 0.6817975044250488, -0.09356460720300674, 0.35459649562835693, -0.022509658709168434, -1.6188124418258667, 0.1379162222146988, -0.8435837030410767, 0.9848524332046509, -0.018466174602508545, -0.6307418942451477, 1.194797396659851, 0.14791373908519745, -1.6127015352249146, -0.17072540521621704, 0.9820282459259033, 1.2810170650482178, -1.1074203252792358, -0.03598969802260399, 0.5199005603790283, 1.7978142499923706, -1.9172176122665405, 1.0055955648422241, -0.9643751978874207, 0.09856110066175461, 1.6010977029800415, 1.319534182548523, 0.20546793937683105, 0.35934558510780334, -1.2936475276947021, -0.5038028955459595, 0.9497313499450684, 2.3399531841278076, 0.5481934547424316, -0.6463760137557983, -0.569838285446167, -0.061814118176698685, -0.8124897480010986, 1.9892690181732178, -0.6936072707176208, -0.5933948755264282, 1.7914429903030396, 1.6697784662246704, 0.19141577184200287, 1.0261870622634888, 1.9669970273971558, 2.0197715759277344, 1.1753878593444824, -0.538581907749176, -0.7411231398582458, 0.2246847003698349, -0.6233555674552917, 1.2220633029937744, 0.8681014180183411, -1.2730426788330078, -0.02240920625627041, -0.3460806608200073, -0.27348756790161133, 0.5756571292877197, -0.2952573299407959, 0.44310417771339417, -0.49149200320243835, 0.016315005719661713, -0.17419220507144928, -1.443163275718689, -0.3309149742126465, -0.08144870400428772, -1.3551714420318604, -1.050033450126648, 5.661954855895601e-05, 1.2353496551513672, 0.8885380029678345, 0.16847869753837585, -1.0912915468215942, 1.271544337272644, 0.014707976020872593, -0.05114922299981117, -0.3378182649612427, -0.8353674411773682, 0.009655131958425045, -0.2577627897262573, 0.5536168813705444, 0.8101122975349426, -1.6241151094436646, 2.7632031440734863, 0.039047472178936005, -1.4984252452850342, 0.09590171277523041, -0.3224436938762665, 0.7590146064758301, 0.5600607991218567, -1.0612746477127075, 0.6752260327339172, -0.5795659422874451, 0.4875926673412323, 0.28281569480895996, 0.7267917394638062, -0.4081396162509918, 0.5946574211120605, 0.03409877419471741, -2.662964344024658, 0.19895760715007782, -1.9252437353134155, -0.9418445229530334, -1.005249261856079, 0.09148408472537994, -0.5702734589576721, -0.9969854950904846, -0.27571016550064087, -1.2468514442443848, 1.1694891452789307, 2.4463226795196533, 0.76944899559021, 0.39653316140174866, 1.1629338264465332, -0.3892679512500763, 0.13868722319602966, 0.10727853327989578, -2.206587314605713, -1.223979115486145, -0.395913302898407, 1.8387196063995361, -0.9997890591621399, 0.8684743046760559, 0.9241253733634949, -0.35637354850769043, -0.40380021929740906, 1.0018144845962524, 1.6053895950317383, -1.9461486339569092, -1.6623940467834473, -0.47318902611732483, -1.4906140565872192, 0.49809619784355164, -1.1129320859909058, 0.6850988268852234, 0.40204864740371704, -0.16688570380210876, 1.1923952102661133, 2.490241050720215, -0.12550857663154602, 0.19549748301506042, -0.5635562539100647, 2.10941481590271, -0.8986755013465881, 1.3758885860443115, 0.30943554639816284, 1.1237735748291016, 0.8075857162475586, -0.3275252878665924, -0.6791381239891052, 0.9703385829925537, -1.2837433815002441, 0.9013701677322388, -1.3807628154754639, -0.2761044502258301, -0.7128746509552002, -0.2495572417974472, -0.2982560992240906, 0.8539493680000305, 1.1276898384094238, -0.048965565860271454, 0.4429647624492645, -1.001198649406433, 0.06849013268947601, -1.2791656255722046, -0.15116268396377563, 0.7801123857498169, 0.7142941951751709, 0.02796461619436741, 1.928636074066162, 0.25282415747642517, 2.293914794921875, 0.8403086066246033, -0.28053152561187744, 1.1959017515182495, 0.20496979355812073, -0.7069832682609558, 0.6753355860710144, -0.5522279143333435, -0.08184362202882767, 0.9411576390266418, 1.2623323202133179, 0.3193351626396179, 1.1157933473587036, -1.3357036113739014, -1.6457158327102661, 0.21922709047794342, 0.42721882462501526, -0.9063215255737305, 0.3929424285888672, 0.21424265205860138, 2.778977870941162, -0.5110279321670532, 0.19178159534931183, -0.980420708656311, -1.2286800146102905, 0.35833683609962463, -1.0191199779510498, 0.7288936972618103, -0.19078265130519867, -1.108233094215393, -2.191277027130127, 1.1682286262512207, 0.5037335753440857, -1.9563795328140259, 2.070605516433716, -0.24767689406871796, -1.2974379062652588, -0.6223503947257996, -1.676377296447754, 0.7935295701026917, -0.12848463654518127, -0.6441071629524231, 0.12079297006130219, -2.0326173305511475, -0.15636543929576874, 2.0449750423431396, -0.6176194548606873, -1.3574925661087036, -1.635934591293335, -0.8314607739448547, 0.7426224946975708, 0.8830327987670898, 1.5238536596298218, -2.152062177658081, -1.5890306234359741, -0.6432729363441467, 0.5732082724571228, 0.04665997251868248, 0.05619080737233162, 1.1310487985610962, -0.49706482887268066, -0.4190012514591217, -1.7731798887252808, -2.41675066947937, 0.4421032965183258, -1.5910826921463013, 0.5528318881988525, -0.08833383023738861, -2.290778398513794, -1.0315700769424438, -0.8754231929779053, 0.535591721534729, 1.8083605766296387, 1.8848909139633179, 0.6947152018547058, 2.502255916595459, -0.2608633041381836, -0.8494582176208496, -1.7643183469772339, 0.8488940596580505, 1.3170585632324219, -0.9964543581008911, 0.3527842164039612, -0.14863942563533783, 0.016275834292173386, 0.4210451543331146, 1.1637413501739502, 1.0661840438842773, 1.8769712448120117, -0.48905086517333984, 0.8603043556213379, -1.9847378730773926, 1.2389835119247437, 0.1078760027885437, 1.1803550720214844, -0.8255777359008789, -1.900457501411438, -0.9782165884971619, 0.24854359030723572, 0.6428680419921875, 0.8012621998786926, 0.30177631974220276, -0.12119720876216888, 1.05013906955719, 0.3352709412574768, -1.6379923820495605, 0.2294454127550125, 0.537162721157074, -0.9611231088638306, 2.197361946105957, -0.720285952091217, -0.6514164805412292, 1.0927139520645142, -0.3935021758079529, 2.283392906188965, -0.10048887133598328, 0.1595863401889801, -0.20501235127449036, 0.9586802124977112, -0.9471454620361328, -0.43182411789894104, -0.5866544246673584, 0.1988437920808792, -0.31668657064437866, 0.33859002590179443, 2.4551689624786377, -0.49233925342559814, -0.6464027762413025, -0.06535890698432922, -0.8329340219497681, 1.2585200071334839, 1.023117184638977, -2.3652517795562744, 1.3534795045852661, -1.191178321838379, -0.11130207031965256, -1.0781950950622559, 0.6791067123413086, 1.5635242462158203, 0.7212203145027161, -1.772241234779358, 0.5730378031730652, -1.1898584365844727, -0.3194217085838318, 0.6377455592155457, -0.19790911674499512, 0.4700046479701996, -0.6374496817588806, -1.222617745399475, 0.9176536202430725, -0.3718459904193878, 0.053064677864313126, 1.281219720840454, -0.2920587658882141, 0.7765706777572632, 1.474921464920044, 1.3918280601501465, 1.2088128328323364, 0.26712915301322937, -1.4564615488052368, 0.19180089235305786, -0.2542695105075836, -0.533860981464386, 0.25318750739097595, 0.0846213698387146, -2.0203185081481934, 0.11328578740358353, 1.0302207469940186, 1.6794782876968384, 1.1937744617462158, 0.08229327201843262, 0.45356228947639465, -0.010169075801968575, -0.8604392409324646, -0.2921507954597473, -0.5807521939277649, -0.8108159303665161, -0.030998526141047478, 0.4033214747905731, 0.0945468619465828, -1.2915184497833252, -0.3907648026943207, -0.04573129117488861, -1.5578999519348145, 1.192863941192627, -1.1556694507598877, -0.04635245352983475, 0.5376834869384766, -1.158131718635559, -0.48036789894104004, -1.389671802520752, 0.2728220820426941, -0.011837530881166458, 0.9220619201660156, 0.8724003434181213, -0.4267599880695343, -0.7984117865562439, 0.0766620934009552, 0.4292966425418854, -1.115733027458191, 1.0413140058517456, -0.486966997385025, 0.423760324716568, 0.09552939236164093, 0.7963305115699768, -0.03540080413222313, 0.4695563316345215, 0.4934198558330536, -1.4589097499847412, -0.19135954976081848, -0.24203187227249146, 0.7840998768806458, 0.0962594524025917, 0.036856602877378464, -1.4029161930084229, 1.2683278322219849, 0.0930122658610344, 0.5607740879058838, -0.9122143983840942, -0.7613030076026917, 1.6577452421188354, -1.1895992755889893, -1.5704010725021362, 0.2194514125585556, -0.47103351354599, -0.7580508589744568, 0.13356725871562958, 1.4617270231246948, 0.6786859631538391, 1.7477641105651855, 0.2692863345146179, 0.17135563492774963, 1.0070388317108154, -0.2035912126302719, 0.1119462326169014, -0.15907227993011475, 0.5374965071678162, -0.8793632984161377, -2.2996344566345215, 0.9130447506904602, -1.2923142910003662, 1.086251974105835, -1.3817371129989624, 0.8721907734870911, -0.05759156122803688, -0.16567479074001312, 0.39613011479377747, 0.10408412665128708, -0.6864613890647888, -1.0100247859954834, -0.22714604437351227, 2.029449939727783, 0.8392744064331055, -1.3404306173324585, -0.8670318126678467, -0.26437222957611084, -0.5255340337753296, -1.1757903099060059, -0.8267340064048767, -1.297356128692627, 1.4723507165908813, -0.7425087690353394, 1.1636008024215698, 0.31610649824142456, -0.4722707271575928, 0.41052544116973877, -1.9765626192092896, 1.9635063409805298, 0.3132498860359192, 0.07825122028589249, -0.3329995274543762, -0.9194794297218323, -0.4875909984111786, -1.4377983808517456, -0.9194694757461548, -0.49936410784721375, 0.9491479396820068, -0.5297530889511108, -0.6052004098892212, -0.015950078144669533, 1.869261622428894, 0.2953147292137146, 0.29864898324012756, -2.4041430950164795, -0.2116091400384903, 1.4666509628295898, -2.163905382156372, -1.5289429426193237, -0.7306970357894897, 0.42177876830101013, -0.15041585266590118, 1.462432622909546, 0.39832308888435364, 1.1926718950271606, 0.7006818056106567, 0.7418850660324097, -0.3215978145599365, 0.991019606590271, 0.27128279209136963, -0.7260189652442932, -2.1484546661376953, 0.20516473054885864, -0.8063772916793823, 0.96453857421875, -0.19931820034980774, 0.6697223782539368, -0.7384045124053955, 0.1987382024526596, 0.1260446012020111, -1.795093059539795, -1.1134504079818726, 1.4385945796966553, -0.905415952205658, 1.0285333395004272, 0.9225567579269409, -1.0585167407989502, -0.23500998318195343, 1.737434983253479, 0.581108808517456, 0.06195446476340294, -0.09669028222560883, -1.5229222774505615, 1.2959479093551636, 0.05915234610438347, 0.0063717663288116455, -0.11508457362651825, -0.6349647045135498, 0.7338443994522095, 0.6196284294128418, -0.23020146787166595, 1.303733468055725, -1.2573193311691284, -0.8196751475334167, -1.9016027450561523, -0.9530235528945923, -1.0829726457595825, 0.5809684991836548, 0.865574300289154, -0.8564289808273315, 1.4688314199447632, 0.9214931130409241, -1.3680087327957153, 0.9265152215957642, -0.27618876099586487, 0.11749359220266342, -1.5397248268127441, -0.47625604271888733, -0.6177856922149658, -1.421420931816101, 0.9165843725204468, 0.5179511904716492, -0.4251880347728729, 2.1549196243286133, 1.1504390239715576, 0.7923431396484375, -0.42202112078666687, 0.4251001477241516, -0.0351729542016983, -1.077620267868042, -0.4692282974720001, -1.438129186630249, -0.19193771481513977, 1.4968675374984741, -0.41469982266426086, -2.368159532546997, -0.9999313354492188, -1.808593511581421, 0.4993914067745209, -0.613304615020752, 0.06002860143780708, 0.836517870426178, -0.517147958278656, -0.9495758414268494, -0.7261948585510254, -0.30322739481925964, 0.4848681688308716, 0.292449027299881, 0.9165636301040649, -0.468373566865921, 0.3090845048427582, -0.8426068425178528, 1.1970270872116089, 0.5840166807174683, -0.5920947194099426, -0.13349954783916473, -0.2677863836288452, 0.4566132426261902, 0.27351054549217224, -0.30221378803253174, 1.362884521484375, -0.8342843651771545, 0.5581320524215698, -2.3956260681152344, 0.7578611373901367, -0.053524043411016464, -0.44889557361602783, -1.0448756217956543, 0.310110479593277, 1.8733829259872437, 0.059555597603321075, 1.9356706142425537, 1.196994662284851, -0.22732719779014587, -1.7045485973358154, -1.8983886241912842, -0.5004746913909912, -1.1722028255462646, -0.18376140296459198, -0.6185693144798279, -0.7987621426582336, 0.28585028648376465, 1.2247477769851685, -0.7987262010574341, -0.7112998962402344, -0.12288353592157364, 1.014156460762024, 1.32694411277771, 0.648668110370636, 0.7915055155754089, -0.6193600296974182, -0.5962168574333191, -0.035024963319301605, -0.9857895374298096, -0.1900116354227066, 1.499989628791809, -0.17033669352531433, 0.8271713852882385, -0.34065231680870056, -1.9864745140075684, 0.06681536138057709, 0.5232593417167664, -0.47862812876701355, -1.2445987462997437, 0.16977499425411224, 0.8957324624061584, -0.34249603748321533, 0.5155420303344727, 1.2438398599624634, -0.19180543720722198, -0.6883550882339478, -0.9415530562400818, 1.5379618406295776, 1.2705570459365845, 1.2947732210159302, 1.3281841278076172, -0.32819005846977234, 0.8863856196403503, -0.7996156811714172, -1.743152141571045, -0.7261773347854614, 0.11794877052307129, 0.42762312293052673, -1.429742693901062, -0.6429134011268616, -0.33942073583602905, 2.0999767780303955, 0.763916015625, -0.705908477306366, 0.6754035353660583, 0.2903880178928375, 0.06316059827804565, 0.030239995568990707, -0.8101581931114197, -1.3633649349212646, -1.6311750411987305, 0.8234719038009644, 0.17252057790756226, -0.8695321679115295, -0.840593695640564, 1.0136252641677856, 1.9429149627685547, -0.9903806447982788], [0.9365671873092651, -0.8741610646247864, 0.08889885246753693, 0.7399998307228088, -1.9516215324401855, 0.5063645243644714, -0.09397701174020767, -1.0195525884628296, 0.25493326783180237, 0.5537213087081909, 1.6965415477752686, 0.9351012110710144, 0.42266470193862915, 1.3458807468414307, 1.0738155841827393, -0.805276095867157, 0.20665541291236877, -0.9506703615188599, -1.077521800994873, -1.0153480768203735, -0.6073523759841919, 0.8688639998435974, 0.2945311665534973, -0.33821815252304077, -0.9464651346206665, -1.7275285720825195, -0.04159107059240341, -0.12930278480052948, -0.7053418755531311, 1.694032073020935, 1.4078997373580933, -0.26331406831741333, -1.9034795761108398, 0.7402182221412659, -1.0491493940353394, 1.648267149925232, -0.6878739595413208, 0.5636191964149475, 0.5221109986305237, 1.023940086364746, 2.456897020339966, 0.17239342629909515, 0.8095871210098267, 0.8342525362968445, 0.5937967300415039, -0.004726271610707045, 0.619155764579773, -1.4233649969100952, 0.13768745958805084, -0.7844000458717346, -1.0133132934570312, 0.3004237711429596, -1.638484239578247, 1.554290771484375, -1.6260943412780762, 0.9732471108436584, 2.2828586101531982, -0.508086621761322, -2.9152822494506836, 0.6561303734779358, 0.29344597458839417, 0.2892993092536926, -0.3488500416278839, -1.0252779722213745, -0.8066809177398682, 1.4982259273529053, -0.06954120099544525, 0.18629102408885956, -1.334336519241333, 0.8277550935745239, 0.5408841371536255, 2.0026254653930664, 0.2555185556411743, -2.9542860984802246, -0.29799261689186096, 0.17789767682552338, -1.6441224813461304, -0.5971950888633728, -1.9359279870986938, -2.1223716735839844, 0.19671975076198578, -0.6240267753601074, -1.0213242769241333, 0.24905985593795776, -0.25670477747917175, 0.8226558566093445, -0.6528086066246033, 0.2635074853897095, -0.8316048979759216, 0.06544853001832962, -1.2056050300598145, -0.8961082100868225, 1.1462291479110718, 2.020354986190796, -0.17556273937225342, 1.6846636533737183, 0.7086048126220703, -0.10476997494697571, -0.7149000763893127, -1.3838677406311035, -0.2610224783420563, -0.0918385460972786, 0.05004618316888809, 0.23324453830718994, 0.37159302830696106, 0.8079272508621216, 0.6635464429855347, -0.018337752670049667, 0.9556031227111816, -1.0198493003845215, 0.14786098897457123, -0.7705823183059692, 0.3899039328098297, -0.07614674419164658, -0.40465500950813293, 0.14929068088531494, -1.9266833066940308, 2.6987786293029785, 0.37807226181030273, -1.646992802619934, -0.6297679543495178, 0.3861715495586395, -1.3322888612747192, -1.184800148010254, 0.2162635624408722, -0.44714033603668213, -0.6406908631324768, -0.38155892491340637, -0.7726092338562012, -0.3188241720199585, 1.4578078985214233, -0.42188963294029236, -0.9577032923698425, -0.4448167085647583, -0.7565410137176514, -0.62355637550354, -0.6465060710906982, 0.7241600155830383, 0.9941018223762512, -0.6617135405540466, 0.36752766370773315, -0.679665744304657, -1.1594839096069336, 0.26169726252555847, -0.4317619800567627, -0.8140199780464172, 0.6244052052497864, -0.4970547556877136, -0.4348568618297577, 0.1894635558128357, -1.1109936237335205, 1.4448953866958618, -0.5102199912071228, 0.20045782625675201, -1.910085678100586, 0.2147967517375946, -0.46002304553985596, 0.0908089280128479, -0.41335973143577576, -0.37344005703926086, 1.3895307779312134, -1.5996731519699097, 0.8467418551445007, -0.7346285581588745, -0.807433009147644, 1.1372482776641846, 0.6597185730934143, -0.029399875551462173, -1.2137503623962402, -0.11986415088176727, 0.8023118376731873, -3.159868001937866, 0.9814608097076416, -0.3948078155517578, 0.019542023539543152, 1.0943870544433594, 0.3529187738895416, 0.16783134639263153, -0.17754581570625305, 0.756703794002533, -0.20766544342041016, 0.35402798652648926, 0.08177347481250763, 1.0792746543884277, -1.462809443473816, 0.7192685008049011, -0.6170298457145691, 0.4555564224720001, -0.6610211730003357, -0.7480571866035461, -1.1829155683517456, -0.6937630772590637, -0.024171041324734688, 0.5726648569107056, -0.3184709846973419, -0.13369779288768768, 1.102068305015564, -0.7814650535583496, 1.3128100633621216, 1.5574735403060913, 1.523751974105835, -0.0026319059543311596, -0.14623422920703888, 0.10163771361112595, 0.562366247177124, 1.0031661987304688, 0.7721545696258545, 0.480155885219574, -0.10707640647888184, -1.4028193950653076, -2.0443615913391113, -0.02978796698153019, -0.3759354054927826, -0.42133572697639465, 1.6566438674926758, -0.35584673285484314, -0.10433468967676163, -2.7859251499176025, 0.706337034702301, 1.2805557250976562, -0.22400496900081635, 0.2642262279987335, 0.5348284840583801, 0.601829469203949, -2.069425106048584, -0.46507200598716736, -0.4420512914657593, -0.3462562561035156, 0.905864953994751, 0.3484317362308502, 0.6044666171073914, -1.170210361480713, 0.3827999532222748, 1.352073311805725, 0.3998565077781677, -0.9456318020820618, 0.017777714878320694, 0.007054501678794622, -1.267014741897583, -0.3473229706287384, -0.0009311396861448884, 0.42681965231895447, 0.49145838618278503, -0.2705247104167938, 0.3821406960487366, 0.5453664064407349, 0.019316358491778374, 1.4504001140594482, -0.20312808454036713, 1.7199938297271729, -1.301272988319397, 0.34784218668937683, -1.2380999326705933, 0.2446238398551941, -0.23306305706501007, -0.5478000640869141, -0.3607245683670044, 1.1879136562347412, 1.5356717109680176, 1.8635733127593994, 0.238427996635437, -0.5286194086074829, 0.05231384187936783, -0.9760003685951233, -0.0042642150074243546, 0.005922598298639059, 0.8293805122375488, -2.1611804962158203, -0.9212360978126526, 0.5783330202102661, 1.0491795539855957, 1.566162109375, -0.2567542493343353, -0.21572721004486084, -0.5492458939552307, -1.6492444276809692, 0.9488444924354553, 0.47494620084762573, -0.4827859699726105, -1.0660127401351929, 2.041750907897949, -0.1401604413986206, -1.176777720451355, 1.6967300176620483, 0.18948113918304443, 0.0032254806719720364, -0.2670744061470032, 0.3423832058906555, -0.01833658665418625, -0.5034288763999939, -0.6067346334457397, 1.2273012399673462, -0.1797667294740677, -0.6576962471008301, 1.9559918642044067, 0.95073401927948, 0.6480895280838013, 0.05708553269505501, -0.44048449397087097, 1.349460482597351, -0.039334360510110855, -0.5656116008758545, 0.7295336127281189, -1.0122319459915161, -0.0014774776063859463, 0.3927719295024872, 1.017169713973999, -0.4104422926902771, -2.129621982574463, -0.08098065108060837, -1.059048056602478, 1.5732393264770508, -0.06385848671197891, 0.3582340180873871, -0.7700785398483276, 1.5032507181167603, 1.8906612396240234, -0.8677860498428345, -0.8831884860992432, 0.8736290335655212, -0.23068852722644806, -0.14262507855892181, -0.20747552812099457, -0.7030705809593201, -0.3070948123931885, -2.3355233669281006, -0.5937468409538269, -1.3355013132095337, -0.1803257316350937, 1.2695140838623047, -1.5635294914245605, 0.03051559440791607, -0.655100405216217, 0.3847874104976654, -0.23892474174499512, -0.365641713142395, -1.3540209531784058, -0.4758279025554657, 0.7497289180755615, -0.48703533411026, -0.367281049489975, 1.3093689680099487, 0.010367054492235184, -0.24136142432689667, 0.04859589412808418, -0.428129643201828, 0.017446357756853104, -0.9866856932640076, -0.43010184168815613, 0.21581239998340607, -0.9345849752426147, 0.6679410338401794, -0.9604939818382263, 0.8823047876358032, -0.8364168405532837, -0.684286892414093, -0.152765691280365, 0.2851797640323639, 0.03441485017538071, -0.873283326625824, -0.35072657465934753, -0.8752045631408691, -0.782204270362854, 0.22797198593616486, 0.7864291667938232, 2.041006088256836, -0.5437509417533875, 1.2112642526626587, -0.9975457191467285, -0.4642084240913391, -1.5887287855148315, -0.6486671566963196, -1.3214507102966309, 1.6685011386871338, 0.33963707089424133, -0.1749822497367859, -1.462791919708252, 0.9383811950683594, 1.301481008529663, -2.2072720527648926, -0.36399316787719727, -1.8717025518417358, 0.7603501677513123, 0.43465888500213623, 1.7135289907455444, 0.04001825302839279, 0.8924994468688965, 0.6960476040840149, -0.5537829995155334, 0.6136586666107178, 1.6988970041275024, 0.2741682231426239, 0.3705940246582031, 0.9832487106323242, 0.7315266132354736, 0.28117287158966064, -0.5316522121429443, 0.348499596118927, 1.130541205406189, 1.4331543445587158, 0.3153132200241089, -0.32420527935028076, -1.2868006229400635, 1.123043179512024, 0.6928268074989319, -0.1925833821296692, 0.9336749315261841, -0.47494006156921387, -1.5617395639419556, 1.6202855110168457, 0.9687450528144836, -0.5798689723014832, 1.1631529331207275, -0.21613353490829468, -0.48333829641342163, 0.27145251631736755, 1.8424912691116333, 1.517053246498108, -0.9268296957015991, 0.01020831149071455, 0.32574912905693054, 0.8910126686096191, 0.04835912212729454, 1.1784164905548096, -0.293487548828125, -0.4020211696624756, -1.054904818534851, -0.6627814173698425, 1.809559941291809, 1.5857499837875366, -0.4106042981147766, -0.01933237351477146, -0.23916654288768768, -0.3487001657485962, 0.39262455701828003, 0.7270883917808533, -0.4732922613620758, 0.21232494711875916, -1.5323907136917114, -0.10402410477399826, -0.3141320049762726, -0.4249604344367981, 1.135132074356079, -0.8707858324050903, -0.34203487634658813, -1.164701223373413, 0.23354631662368774, -0.052145279943943024, -0.6609427332878113, 0.6397601366043091, -1.4754520654678345, -1.7024174928665161, 0.47812899947166443, -0.7939800024032593, 1.376434326171875, 0.6354153752326965, 1.8511741161346436, -0.03647668659687042, -1.2559369802474976, 1.0411723852157593, 1.6360868215560913, 0.04786671698093414, -0.034914083778858185, 0.23101261258125305, 1.4783098697662354, -1.8880105018615723, -1.2240347862243652, 0.23295961320400238, -0.2919453978538513, 0.3010869324207306, -0.48459383845329285, 0.3425799310207367, 0.3324373662471771, 0.28766947984695435, -1.607106328010559, 0.00920599140226841, 2.00278902053833, 0.26326820254325867, 0.019005995243787766, -2.0404555797576904, 0.7883474230766296, 0.3239777088165283, 1.7998428344726562, 1.5287023782730103, 1.4108753204345703, -1.6395282745361328, 0.5147484540939331, -0.617038369178772, -1.0883336067199707, -2.7574963569641113, -0.017999423667788506, -0.7074428200721741, -0.3258107006549835, 1.3561700582504272, 0.409061998128891, 0.09748614579439163, 1.083482265472412, 0.44194522500038147, 1.4432967901229858, 1.1674494743347168, 0.579887866973877, -0.3708189129829407, -0.6571645736694336, 0.6255336403846741, 0.201907679438591, -1.1969542503356934, 0.4325186014175415, 0.5710278749465942, 0.22031623125076294, -0.7753495573997498, -0.1754535585641861, -0.7213848829269409, 2.158994674682617, 0.6952947974205017, 0.0065941656939685345, 4.260534286499023, -0.5179327726364136, 0.8136230111122131, 0.3463003933429718, -0.418508917093277, 0.7820060849189758, 0.812775731086731, 0.7078719735145569, -0.17975622415542603, -0.7299095392227173, 0.28194260597229004, 1.2289150953292847, 1.4481748342514038, -1.2411125898361206, -1.6241402626037598, -0.2125971019268036, -1.4141408205032349, 0.7702990174293518, 0.9593429565429688, -0.8613577485084534, 0.21608611941337585, 1.3225277662277222, 0.271506667137146, 1.222470760345459, -1.5239825248718262, 1.8644740581512451, 0.39009764790534973, -0.7218343615531921, 1.4135687351226807, 1.6234962940216064, 0.004707171116024256, -0.35515138506889343, 0.6716257333755493, -0.9246966242790222, -0.38355061411857605, 0.8592363595962524, -1.4130042791366577, -0.6987966895103455, 0.1794530302286148, -0.2796247899532318, -0.008381913416087627, -1.0831198692321777, 0.9915074110031128, -0.14825467765331268, 1.478432297706604, -1.6346255540847778, -0.015168274752795696, 0.5811720490455627, -1.2896289825439453, 0.998733401298523, -1.7330743074417114, -0.44394415616989136, -0.5290526151657104, -0.5543021559715271, 0.8150860071182251, -0.6618127226829529, 1.586794137954712, 0.18940207362174988, 0.5659103393554688, 0.042376283556222916, 1.1884318590164185, -0.20403194427490234, -0.434919536113739, -0.2344367355108261, 0.9049422144889832, 0.2785906493663788, -0.5380187034606934, -0.038628075271844864, -0.34645840525627136, -0.1341894268989563, 0.36363235116004944, -1.0991008281707764, 1.8034133911132812, -0.5280172824859619, 0.02175275795161724, 1.5961990356445312, 1.3439775705337524, -0.45993608236312866, 0.7077053785324097, -1.0195693969726562, 0.2194667011499405, 0.6853396892547607, 1.54529869556427, -2.5552358627319336, 0.25694113969802856, 1.6115044355392456, 0.9252454042434692, -1.1626334190368652, -1.9589730501174927, -0.07232838124036789, 0.9936947226524353, 1.7455272674560547, -0.8538271188735962, -0.611057460308075, -0.08753903955221176, 0.46387046575546265, 1.1640208959579468, -0.15283630788326263, 0.22261200845241547, -0.41860243678092957, -0.3563379645347595, 0.6580312848091125, -0.405274897813797, 0.3832612931728363, 0.3158334195613861, -0.422598272562027, -0.43066954612731934, 0.08967237174510956, 0.014570379629731178, -0.5094195008277893, -0.5486961603164673, -0.1035730242729187, -1.4313886165618896, -0.7143517732620239, -0.8745594620704651, 2.319707155227661, 0.454615980386734, -2.2211272716522217, 0.1478029191493988, -0.4473775625228882, 1.301815390586853, 0.677095353603363, -1.2164325714111328, 0.31972914934158325, -1.3561345338821411, 0.9374086856842041, -0.7219738960266113, 0.1276877224445343, -2.01997971534729, -0.26281678676605225, -1.7065720558166504, 0.2899952828884125, -1.6070829629898071, 0.11527425795793533, 2.502809524536133, 1.0905247926712036, 0.6869838237762451, 0.4682926833629608, -1.5845245122909546, -0.8410924673080444, 0.26705604791641235, 1.3620834350585938, 1.9582061767578125, 0.8613343238830566, 0.1619061827659607, -0.29242274165153503, -0.15127456188201904, 0.430353045463562, 1.1640311479568481, -0.21257919073104858, 0.5642008781433105, 0.9832409620285034, -0.8394498229026794, -1.152960181236267, 1.642501711845398, -1.3846865892410278, 0.9163469672203064, -0.20310355722904205, 1.3098621368408203, -1.558335304260254, 1.4232548475265503, -0.8390951156616211, -0.3365904688835144, -0.7512370347976685, -0.5034655332565308, -0.0502341091632843, 0.8646107912063599, -0.22958339750766754, -0.8691819906234741, 2.1791577339172363, -1.9513195753097534, 1.663893699645996, 0.13299022614955902, 1.602582573890686, -1.244667649269104, -0.3474526107311249, -2.644584894180298, 0.12071531265974045, -1.2314352989196777, 0.3657605051994324, 0.09959758073091507, 0.5985063910484314, 0.07689273357391357, -0.25538110733032227, 1.3598203659057617, 0.13209596276283264, 0.9804049730300903, -0.888696551322937, 0.6941645741462708, 0.7851325273513794, -0.010547705926001072, 1.4089549779891968, 1.4566593170166016, -0.782545268535614, -1.2835742235183716, -0.6176347732543945, 1.3975192308425903, -0.04131714254617691, -0.3526832163333893, 0.5296095013618469, -0.11795228719711304, 0.40842539072036743, -1.3078668117523193, 0.16571809351444244, -0.06933135539293289, -0.1515100598335266, -0.9357810616493225, -1.1763896942138672, -1.1372230052947998, 1.23251473903656, -0.1284172087907791, 0.07326947152614594, 0.8782618641853333, -0.3242948353290558, 0.03139748424291611, 0.8749449849128723, -1.2889165878295898, -1.4495726823806763, -0.5229372978210449, 0.9272090196609497, -0.6761972308158875, -0.0461580865085125, 0.45887675881385803, -0.2485627382993698, 2.0428950786590576, -0.8802183270454407, -0.08387420326471329, -0.45682376623153687, 0.6611866354942322, 0.8104873895645142, -2.146573781967163, 0.395713210105896, 0.5951042175292969, 0.054713174700737, -0.5272396802902222, -1.1961970329284668, -0.515599250793457, 1.1298160552978516, 0.5708828568458557, -0.9379553198814392, -0.2966446280479431, 0.020672043785452843, 0.4486190676689148, -0.29130280017852783, -0.28408873081207275, 0.9187955856323242, 0.7597159147262573, 0.08292146027088165, 0.877373456954956, 0.21071308851242065, -0.5163393020629883, -2.035294532775879, -0.7720992565155029, 0.7409964799880981, 0.4103573262691498, 0.10837343335151672, -1.2802752256393433, -3.076320171356201, 0.7707753777503967, -0.13361519575119019, 0.39685967564582825, -0.2681010663509369, -1.0338841676712036, 1.8693263530731201, -1.0834599733352661, -0.1573752760887146, 0.7393395304679871, -1.3357406854629517, 0.30259472131729126, 0.5404074192047119, -0.5638326406478882, -0.06784020364284515, -0.37494179606437683, 1.0847951173782349, -0.7208128571510315, -1.2435903549194336, -0.02213754877448082, -0.8900061845779419, -0.3092460036277771, 0.18405522406101227, -0.42746782302856445, -1.549169659614563, -0.8547729253768921, 0.16810958087444305, 0.5449604988098145, -0.5229714512825012, -0.08434224873781204, -0.15575911104679108, -0.15938343107700348, 0.16181890666484833, -2.020914077758789, 0.46816447377204895, 0.5143834948539734, 0.5198489427566528, 0.222226083278656, -0.5957174301147461, -1.741079330444336, 0.8587823510169983, -0.2720382511615753, 1.4038071632385254, -0.5401888489723206, -0.6762743592262268, 0.6331536769866943, 1.0920653343200684, -1.2426087856292725, 0.5190433263778687, 0.42932915687561035, 0.2986115515232086, 0.29857343435287476, -1.5512267351150513, 0.06983573734760284, 1.4487085342407227, 0.3386153280735016, 0.7915337085723877, -0.7643839716911316, 0.48164835572242737, -0.31791090965270996, -0.692599356174469, -0.6585453152656555, -0.8641418814659119, -0.5785174369812012, 0.5550187826156616, 0.17003436386585236, -0.6430851221084595, 1.7583434581756592, 0.9562125205993652, 1.4664878845214844, 1.3162856101989746, -1.1173027753829956, 0.21693751215934753, 0.32363787293434143, 0.7874605059623718, -0.32585960626602173, 0.16074855625629425, 0.7820441126823425, -2.8017473220825195, -1.9473323822021484, -2.6140623092651367, -1.9303679466247559, -0.8785800337791443, 1.004026174545288, -1.0206304788589478, -0.6023805141448975, -0.24924834072589874, 0.9032645225524902, 1.2375715970993042, -1.2489374876022339, -0.30354756116867065, 1.2846479415893555, 1.4968206882476807, -0.3731950521469116, 0.7265489101409912, -0.03771061450242996, 0.37024056911468506, 0.11507757008075714, -0.08144565671682358, 0.4738391935825348, 1.234823226928711, -1.1665046215057373, 0.9438257217407227, -0.035773780196905136, 1.2141820192337036, 0.6498830914497375, -0.3103594481945038, 0.16100876033306122, -0.6144925355911255, 0.5312967300415039, -0.6673086285591125, -0.20690301060676575, 0.1019882783293724, -0.7956096529960632, -0.1491921842098236, 1.6736096143722534, -0.10741571336984634, -0.370343953371048, 1.89556884765625, 0.649789035320282, -0.2019939422607422, -0.05873963609337807, -0.06713468581438065, 1.9767429828643799, -0.8584657311439514, 0.7409657835960388, 1.6485486030578613, -0.9144847989082336, 0.6367045640945435, 1.3773823976516724, -1.2263221740722656, -1.6258280277252197, 1.9786937236785889, -0.08194147050380707, 0.7537388801574707, -0.8560123443603516, -0.7504592537879944, 0.48577794432640076, -0.2532682716846466, 0.9911214709281921, -0.4364061951637268, 0.012425067834556103, -1.608971357345581, 1.090802788734436, 0.30137619376182556, 1.0775858163833618, 0.8221520781517029, -1.1153484582901, 2.1472434997558594, 0.47434085607528687, -0.4432826340198517, -0.1833353489637375, -0.703400731086731, -1.5315908193588257, -0.6839354038238525, 0.6681466102600098, -1.4379266500473022, -0.8571068048477173, 0.06524059176445007, -1.484266996383667, 0.1401216983795166, -1.647632360458374, -0.13820421695709229, -0.3639943599700928, 0.059043850749731064, 0.7738882303237915, -1.8033074140548706, -0.6564435958862305, -0.3012215495109558, -0.7350355386734009, -1.6742135286331177, -1.9435160160064697, 0.007391761057078838, -1.0817844867706299, 0.5175004601478577, 1.100600004196167, 0.043331075459718704, -0.5153324007987976, -0.37033405900001526, -0.4849373400211334, -0.44141310453414917, -0.08187796175479889, 0.9104294180870056, -0.23875032365322113, 0.7902103066444397, -1.5155295133590698, -0.5636088252067566, -0.024746477603912354, 2.0810704231262207, -1.077296257019043, -0.8611156940460205, -0.3687095642089844, -0.1574040651321411, -1.1012864112854004, -1.549547791481018, -2.1043949127197266, 0.8054923415184021, 1.463020920753479, -0.6650912165641785, 1.095658540725708, 0.583244264125824, -0.33820998668670654, 0.9863576292991638, 0.398360013961792, -1.4348217248916626, 0.9857897758483887, -1.3436754941940308, 1.786977767944336, -0.015690796077251434, 0.6257538199424744, -1.4970932006835938, -1.601293683052063, -1.1417553424835205, -0.48171353340148926, -0.8791269659996033, 0.4775277078151703, 0.09941521286964417, 1.343356966972351, 0.6720564365386963, 0.7613542079925537, -0.35726016759872437, -1.104500412940979, -1.1773908138275146, 0.6464260220527649, -0.7160065770149231, -0.7272130250930786, 0.05843142047524452, 0.5465442538261414, -2.472527027130127, -1.432874083518982, -0.26608580350875854, 1.6134533882141113, -1.1492379903793335, -0.6238918900489807, 1.128889799118042, -1.4606868028640747, -1.368361234664917, 0.5048437714576721, -1.8302563428878784, -0.07809989154338837, 0.40760791301727295, -1.1780527830123901, 0.942764937877655, -2.020897150039673, 0.3257211744785309, 1.0501407384872437, -0.7064634561538696, 0.5014256834983826, 0.04486823454499245, 0.211800679564476, -0.07612475007772446, -0.3897428512573242], [1.5790410041809082, -0.5333259701728821, 0.11988304555416107, 0.8516451120376587, -1.4689759016036987, -0.45922762155532837, 0.40135571360588074, 0.6347907185554504, 0.7584517002105713, -2.0185179710388184, 0.056832242757081985, 0.05274377763271332, -0.4421713352203369, -0.4269985258579254, 1.1759146451950073, -1.4855804443359375, -0.6931820511817932, 2.58648681640625, 0.03250088542699814, 0.06461571902036667, 3.0376689434051514, -1.5153788328170776, 0.8900222778320312, 1.003725290298462, 1.3549935817718506, -0.7745527029037476, -1.2698343992233276, -1.4044948816299438, -0.16777874529361725, -0.3005048930644989, 0.3490322530269623, -0.43632909655570984, -0.6438360810279846, -0.3777082860469818, -1.1536246538162231, 1.1329306364059448, -0.004348237998783588, 1.3907966613769531, -0.5396814942359924, 0.42701107263565063, -0.40826690196990967, -0.008467521518468857, -0.0005789831047877669, -0.22256696224212646, -0.9041277766227722, -0.2674649655818939, -2.116576910018921, 2.0015065670013428, -1.451448678970337, 2.028437376022339, 3.884753704071045, -0.07533115148544312, -2.571682929992676, 0.15980301797389984, -0.04246993735432625, 1.8240762948989868, -0.6051117181777954, 0.5947074294090271, -0.6864020228385925, -0.1361149698495865, 0.9191280007362366, 1.3895902633666992, -0.10857026278972626, -0.587679386138916, -1.4422909021377563, -1.3631830215454102, 0.32509058713912964, -1.0634384155273438, 0.9036625027656555, -1.3088266849517822, 0.5810183882713318, -1.6032617092132568, -0.13909146189689636, 0.08269382268190384, -0.5580311417579651, 0.2977522909641266, -1.3517106771469116, 1.379982352256775, 0.34340953826904297, -0.8451955318450928, -1.1272605657577515, 0.31637200713157654, -1.5741674900054932, 1.992500901222229, 0.026547854766249657, -0.05228458344936371, 1.5042834281921387, 0.056602843105793, -1.0710715055465698, 0.49221011996269226, 0.08877457678318024, -0.349911093711853, 1.149573564529419, 1.8468290567398071, -1.2035266160964966, -1.682936668395996, -0.772987425327301, 0.4155147969722748, -0.6701183915138245, 0.8344398736953735, -0.1314055174589157, -0.5145995020866394, 0.872635006904602, -0.8309293985366821, -0.6138731837272644, 0.8426862955093384, 0.6840877532958984, -0.26184695959091187, 1.1380181312561035, -0.990861713886261, -0.30834507942199707, 1.4471582174301147, 1.67621648311615, 0.7964335083961487, 0.21899302303791046, 0.08270079642534256, 1.437233805656433, 0.8532575368881226, 1.4274581670761108, -0.1841527670621872, 0.39790040254592896, 0.42542025446891785, 0.431327223777771, -0.16046807169914246, -0.496616393327713, -0.33660587668418884, 0.2564760744571686, 0.10143870115280151, -0.634897768497467, -0.3264676332473755, -0.4897761046886444, -0.5421111583709717, -1.1904144287109375, -0.8648845553398132, 1.2109739780426025, 0.06788090616464615, -0.4283689260482788, 0.9268468618392944, -0.6033276319503784, 0.1771465390920639, 0.9869987368583679, -2.7483716011047363, 0.20848721265792847, -0.5264684557914734, -1.5718374252319336, 1.2013715505599976, -1.088767170906067, 0.8428657054901123, -0.4259011447429657, 0.3521260917186737, -0.07801006734371185, 0.9391900300979614, 0.7458517551422119, 0.6606478691101074, -0.6090106964111328, 0.8929362297058105, -1.2193663120269775, 0.484920471906662, -0.555679976940155, 0.3112418055534363, -0.18940943479537964, -0.6730515956878662, -0.1303275227546692, 1.6589447259902954, -0.9171041250228882, 1.8185484409332275, -0.36193808913230896, 1.7556822299957275, -0.0183245912194252, -0.8577933311462402, 0.10502633452415466, 0.14103755354881287, -0.574079155921936, 0.48395606875419617, 1.4566383361816406, -0.2731565833091736, -0.35736435651779175, 0.7866747379302979, 0.1379261016845703, -1.6936187744140625, -0.8517091274261475, 0.854108452796936, 0.06820839643478394, -1.4921694993972778, 1.4840248823165894, 0.2725348472595215, 0.5305510759353638, -1.7560553550720215, 0.2630701959133148, -2.4295918941497803, -1.4101732969284058, -0.7554728984832764, 0.44537976384162903, 0.8639181852340698, 1.0228644609451294, -0.4729546010494232, -0.31592264771461487, 0.032210443168878555, -0.8214594721794128, 0.9463706612586975, -1.228774070739746, -0.6353492736816406, -0.8459019064903259, 0.14967885613441467, -1.3707646131515503, -0.34978893399238586, 2.1327931880950928, 1.5160740613937378, -0.14698030054569244, -1.1873071193695068, 1.1682063341140747, 0.008300951682031155, -0.6779213547706604, -0.6231728196144104, 0.6035215854644775, 0.8889107704162598, -0.8086427450180054, 1.3319404125213623, 0.2050093710422516, 0.2532205283641815, -0.5655535459518433, 0.5011729598045349, 0.28011101484298706, 0.1836872100830078, -0.028993656858801842, 0.719967246055603, -1.2229026556015015, 2.318129539489746, -0.008816620334982872, -1.2080413103103638, 2.0237932205200195, 0.2869757115840912, 0.5474008321762085, 0.4711553752422333, -0.7506442070007324, -0.2887071669101715, 0.12640148401260376, 1.1492385864257812, 0.14242587983608246, 0.6629072427749634, -2.196190357208252, -1.7892740964889526, 0.56160569190979, 0.896489679813385, -0.24575559794902802, 0.45503780245780945, -1.3946880102157593, -0.14171753823757172, 1.201987624168396, -0.2739074230194092, -0.719904899597168, 0.822094202041626, -0.45733100175857544, 0.6033405065536499, -0.024851975962519646, 0.32853540778160095, -2.783029556274414, 1.838372826576233, 0.05699344724416733, -0.5459349751472473, -0.9661317467689514, 0.0712323933839798, 0.6055536270141602, -0.44203925132751465, -0.15328234434127808, -0.9859053492546082, 1.598402738571167, 0.5539939403533936, 1.5033276081085205, 0.5190866589546204, 0.4210077226161957, -0.21849386394023895, -0.8867558240890503, 1.6001299619674683, 1.0832017660140991, 0.8710755109786987, -0.6183202266693115, -0.8395958542823792, 0.9606888890266418, -0.32922014594078064, 0.7749538421630859, -2.345755100250244, -0.879404604434967, -0.2504173517227173, -2.374924421310425, -1.168796181678772, 1.8737727403640747, -1.1107856035232544, -1.4863933324813843, -1.2079647779464722, -0.6569646596908569, -0.9482342600822449, -0.8040580749511719, 0.38465484976768494, -0.7499444484710693, 0.45051223039627075, 0.9065628051757812, 1.195115089416504, -1.0147041082382202, 0.07714339345693588, -0.26369509100914, 0.7244055271148682, -0.60737144947052, 2.5126900672912598, 0.06380384415388107, 0.0742967426776886, 0.8204240202903748, -0.384703665971756, -0.5411695241928101, 0.4709254503250122, 0.47609323263168335, 0.5264269709587097, 2.0650534629821777, 0.5718123912811279, 1.611211895942688, -0.8369691371917725, 0.8701989650726318, 0.5348974466323853, 0.5626625418663025, 1.8904101848602295, 2.3754587173461914, 0.10628514736890793, 1.7967329025268555, -0.49832209944725037, -0.4517481029033661, -0.32829636335372925, -0.19797132909297943, 0.3630640208721161, -0.11051469296216965, -2.03635835647583, 0.40749990940093994, 0.579729437828064, -0.964381754398346, -0.4241981506347656, 0.7773828506469727, 0.2836589813232422, -0.6490817070007324, -0.7138742208480835, -0.4893364906311035, 0.9517606496810913, -1.0471580028533936, -0.33491042256355286, -1.6111208200454712, -1.238587498664856, -1.2915403842926025, 1.0299352407455444, -0.469046026468277, 0.2657644748687744, 0.4993973672389984, 1.2846952676773071, -0.8777210116386414, 0.32998278737068176, -0.056699030101299286, 0.7146924138069153, 0.8551108837127686, -0.35825657844543457, 0.0029907829593867064, 0.47225314378738403, -0.9169167876243591, 0.630547046661377, -0.3004702925682068, 0.3899956941604614, 1.1572260856628418, 2.051257371902466, 0.2261696308851242, -0.3458872437477112, -2.064969778060913, -1.980887532234192, -1.0321365594863892, -0.9059420228004456, -0.4536183774471283, 0.24536187946796417, 0.972236692905426, -0.6274819374084473, -0.42699044942855835, -0.43694940209388733, -0.0023094876669347286, 0.04850190505385399, 1.1967228651046753, -1.071626901626587, 0.10650823265314102, -0.16947674751281738, 1.111230731010437, -0.46087557077407837, -0.7060760259628296, -0.9631352424621582, 0.21336226165294647, 1.0831648111343384, -0.7746662497520447, 0.13652990758419037, -3.085324287414551, -0.29283466935157776, -1.4952960014343262, 0.4907463788986206, -0.997298002243042, -2.3057312965393066, -0.5378188490867615, 0.6232960820198059, 1.7240464687347412, 0.050636716187000275, 0.6298943161964417, 0.46450141072273254, 0.536976158618927, 0.635694146156311, -0.2054986208677292, 0.4186953604221344, 0.1312110722064972, -0.20507097244262695, -0.7914811968803406, -1.5867184400558472, 1.4130467176437378, 1.641711950302124, -0.9340488910675049, -0.7045261263847351, -1.2631100416183472, 2.3857598304748535, 0.34036779403686523, 0.4373723268508911, -0.5559303164482117, 0.995002269744873, 1.1275336742401123, 0.10894399136304855, -1.7676973342895508, -0.2644392251968384, 1.8885374069213867, 0.0768066868185997, -0.9547189474105835, -1.2185412645339966, 0.11468878388404846, 1.917556881904602, -1.4023807048797607, 0.30850744247436523, -0.5259387493133545, -0.10377446562051773, 0.7838740944862366, 0.14797072112560272, -1.3331260681152344, -0.0754571184515953, 0.957763135433197, 1.172765851020813, 0.14817479252815247, 1.450901985168457, -0.014489976689219475, -0.48511525988578796, -0.35282817482948303, 0.14766642451286316, 0.7645722031593323, 0.51521897315979, 0.555561900138855, -1.009744644165039, -0.8737872838973999, 0.34051406383514404, -1.0861812829971313, 0.913343608379364, -0.2785831391811371, 0.2760365605354309, -0.30951476097106934, 0.7391947507858276, 2.7020480632781982, 1.0052438974380493, 1.246084213256836, 0.0032620502170175314, -0.3051169514656067, 2.0027432441711426, 1.0085290670394897, -0.733526885509491, 0.2024131864309311, 1.0794986486434937, -0.5426211953163147, -1.4315568208694458, -1.7863517999649048, -0.7365837693214417, -0.6228170990943909, 0.21762803196907043, 0.3672483563423157, -1.0239636898040771, 0.3385658860206604, 1.6497677564620972, 1.422348976135254, -0.7743648886680603, 0.1891166716814041, -0.011799710802733898, -0.8648548722267151, 0.8792310953140259, -0.36522310972213745, -0.7231929302215576, -0.7426613569259644, 1.5290887355804443, 1.0032377243041992, -1.083957552909851, 1.6246532201766968, -1.4751780033111572, 0.6696956753730774, 2.401149272918701, -0.8618019223213196, 0.25953590869903564, -0.3058164417743683, 0.5559561252593994, -0.6295157074928284, 0.4427540898323059, -0.16724392771720886, -1.030940055847168, 0.7912148237228394, 0.04536418989300728, -1.206675410270691, 0.9335427284240723, 1.8443711996078491, 0.7755793929100037, -1.3114162683486938, 0.31560400128364563, 0.4625851511955261, -0.7936951518058777, 0.473974347114563, -0.4894402027130127, 1.280479073524475, 0.031135916709899902, -1.7382289171218872, 0.5043020248413086, -1.1824302673339844, -1.2959668636322021, -0.12690097093582153, -0.5487714409828186, 0.23783785104751587, 1.62077796459198, 2.9771006107330322, -0.1617571860551834, 0.5321395397186279, -0.019910765811800957, -0.45807430148124695, -1.2046630382537842, 0.6560438871383667, -0.06679477542638779, 0.13564515113830566, 0.33245164155960083, 1.0633598566055298, 0.6216594576835632, -1.1073967218399048, -0.3901579678058624, 0.8364368677139282, -0.45172515511512756, -0.9902114868164062, -0.765986979007721, 0.340889036655426, 0.7087827324867249, -2.5289597511291504, 1.726457953453064, -0.0575571246445179, -1.4628263711929321, -1.4481016397476196, -1.3479489088058472, 2.0537922382354736, -1.169530987739563, 0.998300313949585, -2.334811210632324, -1.556846261024475, 1.4745124578475952, -0.13367028534412384, 2.0073156356811523, -0.251971572637558, 0.8704094290733337, -0.20995190739631653, 1.2125325202941895, -0.009220282547175884, 1.2751185894012451, 0.9088345766067505, -1.5746567249298096, -0.29913368821144104, -0.7863946557044983, 0.7625253796577454, 0.1900131106376648, 0.20770888030529022, 0.29205745458602905, 0.019425464794039726, -0.5018163323402405, 0.902607798576355, 1.3517320156097412, 0.9005956649780273, 0.3932807743549347, 1.541292667388916, -0.7199041843414307, 1.4296458959579468, -0.19819824397563934, -1.105627179145813, 0.0704704076051712, -0.6159945130348206, 0.5116443634033203, -0.3047895133495331, -0.9421954154968262, 0.493828684091568, 0.4729937016963959, 2.0319437980651855, 0.1873423308134079, -0.304293692111969, 0.12027760595083237, 2.2739412784576416, -0.5993536710739136, -1.0057318210601807, -0.6110113263130188, 1.797369360923767, 0.17884571850299835, -2.2917892932891846, 1.4913625717163086, 1.3393633365631104, 1.0742993354797363, -1.9445602893829346, -0.2892419993877411, -0.7264238595962524, 0.285209596157074, 0.8585309982299805, -0.16521751880645752, -0.06599382311105728, 0.1815684288740158, 1.5979571342468262, -0.7729672789573669, 1.276652216911316, -0.07993046939373016, 2.4277639389038086, -0.4622264802455902, 0.1883525401353836, 0.4264324903488159, -1.7585452795028687, 1.027099609375, 0.8435952663421631, 0.2359752655029297, -0.4480692744255066, 1.4248311519622803, -1.2260273694992065, -0.03308872878551483, 1.375074863433838, -0.02158374711871147, 0.47496697306632996, 0.3809342384338379, 0.05039630085229874, -0.7182119488716125, 0.49455010890960693, 1.0596522092819214, 1.5545873641967773, -1.7416529655456543, -2.0887451171875, -0.6200222969055176, -0.7768572568893433, -0.6423871517181396, -0.35895946621894836, 0.42191386222839355, -0.39257001876831055, 0.332643061876297, 0.5980106592178345, -0.5303933620452881, 0.5264252424240112, -0.06068127974867821, 1.3875517845153809, 0.17965830862522125, 0.23180963099002838, -1.5806682109832764, -0.375906765460968, -1.1933043003082275, 0.2844281494617462, -0.4298885762691498, 0.13687428832054138, -0.9303194284439087, 0.6407140493392944, -1.4079817533493042, -1.8637505769729614, 1.0672115087509155, 0.5730065703392029, 0.7583123445510864, 0.15679191052913666, 1.0438343286514282, 0.8970044255256653, -1.4632869958877563, -0.7216981649398804, -0.5962018966674805, 0.005557643249630928, -0.11356379836797714, -0.909977376461029, -0.67722487449646, -1.6336045265197754, 0.24670739471912384, -0.11194673925638199, -1.154896855354309, -1.3537623882293701, 1.3976398706436157, 0.43244999647140503, 0.8966178297996521, 1.7128280401229858, 0.32837799191474915, 0.8488338589668274, -0.2092621922492981, 0.8462848663330078, -1.5767773389816284, 2.998842477798462, -0.5824215412139893, -0.7852509617805481, -0.26537397503852844, 0.8300455808639526, 0.5695860385894775, 0.721210241317749, 0.4809720814228058, 0.037306345999240875, -1.2772257328033447, -1.8413125276565552, 0.598675549030304, -0.19147273898124695, 1.2208069562911987, 0.22395411133766174, 0.7462950348854065, 1.6315288543701172, -0.8726938962936401, -2.2879035472869873, -0.44806724786758423, 0.8181455135345459, 1.4961942434310913, -0.6235970854759216, -0.6201680302619934, -0.7470925450325012, -0.4295324981212616, 0.6883578300476074, 0.1811814308166504, 0.7703401446342468, 0.02588902786374092, 0.6653184294700623, 0.0586387999355793, 0.39027291536331177, -1.737135648727417, 0.5477546453475952, 0.051974475383758545, 0.1919701099395752, 0.736614465713501, 2.404836893081665, 0.08139607310295105, -0.5568199753761292, 0.09941501170396805, -1.3893115520477295, -0.03093208745121956, 0.31460297107696533, 1.595544457435608, -0.5896806120872498, -1.4445390701293945, 0.4983094334602356, -0.03227442875504494, -1.4817692041397095, 0.9094472527503967, 1.0337650775909424, 0.24476973712444305, 1.144864559173584, 0.18846061825752258, -0.9216044545173645, -0.8909235000610352, 0.20356003940105438, -0.7461983561515808, -0.15033569931983948, 0.8784538507461548, -0.5313497185707092, -0.9436431527137756, 0.6616905927658081, 0.6414148807525635, 0.39072152972221375, -1.0517539978027344, -2.5113959312438965, 0.46989330649375916, 0.5013938546180725, -0.23349812626838684, -0.9741975665092468, -0.14181828498840332, -1.2214637994766235, 0.2064765989780426, -0.23599915206432343, -0.7315207719802856, -1.6714227199554443, -2.213059186935425, 0.5701503753662109, -0.3039351999759674, 1.202385425567627, -1.3839943408966064, -1.4876224994659424, 0.3535534143447876, -0.47529393434524536, -0.0425230972468853, -0.08170872181653976, 1.0642735958099365, -0.9523839950561523, -0.8561050891876221, 1.8721356391906738, -0.4022867679595947, 0.13327474892139435, -0.5116124153137207, 0.04914761334657669, -0.4414624869823456, -1.450645923614502, -0.5705547332763672, -0.8570750951766968, -0.3582107126712799, -0.2922869324684143, 2.3020710945129395, 0.8590265512466431, -0.8804674744606018, -0.8294894099235535, 0.6893266439437866, 0.9134179353713989, -0.01220277976244688, 1.5292205810546875, 0.21030525863170624, -0.6058949828147888, 0.276170939207077, 0.6012341380119324, -0.6968925595283508, 0.5281249284744263, 1.6780948638916016, -1.409847378730774, -0.1424250304698944, 0.2020142376422882, -0.06723669171333313, 0.5267544388771057, 0.09607692062854767, -1.4322603940963745, 0.4363335967063904, -0.8375811576843262, -0.9962500333786011, 0.09720011800527573, -0.6469987034797668, 0.3591514825820923, -0.904772162437439, -0.7367318868637085, -0.24469244480133057, -0.20244817435741425, -1.6331226825714111, -0.34675145149230957, 0.2614492177963257, -0.07628077268600464, -7.611006731167436e-05, -0.35052400827407837, 0.7500158548355103, 0.9220948219299316, 1.014726161956787, 0.6826624870300293, -0.21370184421539307, -0.07886043190956116, 0.759706974029541, 0.04192572087049484, -0.44884371757507324, -0.7753785848617554, 0.7126299738883972, -0.3740050196647644, 0.4716605842113495, 0.8392072319984436, -0.43332239985466003, 0.5800988674163818, -0.3676232397556305, -0.19766873121261597, 1.7331515550613403, 0.1720404475927353, 0.4804937541484833, -1.4025132656097412, 0.8830023407936096, 1.9756687879562378, 1.2665843963623047, 0.5461181402206421, -0.8620558977127075, 0.3498097360134125, 0.17886073887348175, 0.10766562074422836, 0.36952200531959534, -0.31568920612335205, -1.29172945022583, 1.2213330268859863, -1.872768759727478, -0.3427763879299164, 1.3692729473114014, 0.499957412481308, 0.5881775617599487, 1.3381547927856445, -0.880537211894989, -0.9715482592582703, 0.5820170044898987, -1.4735058546066284, -0.18791887164115906, -0.5012770891189575, -1.7290233373641968, 0.9378520250320435, -0.7842134237289429, -1.0150772333145142, 0.667976975440979, -1.154801368713379, -0.3444253206253052, -0.004378488287329674, 0.9263703227043152, -0.2011413276195526, -0.7564501762390137, -0.3408205807209015, -1.3653359413146973, 1.2385222911834717, 1.9301565885543823, 0.45645496249198914, -0.9762454628944397, 0.7722456455230713, -0.7349080443382263, 0.757338285446167, 0.026009153574705124, 0.16145695745944977, -0.563022792339325, -2.136353015899658, -0.8779847621917725, 0.5471293330192566, -0.2645719051361084, 0.31773966550827026, -1.1838090419769287, -1.3384172916412354, 0.9857520461082458, 1.601987361907959, -0.4218643307685852, -0.7957558035850525, -0.30362391471862793, -0.7244424223899841, 0.44273850321769714, 1.6472082138061523, -1.1038668155670166, -0.4007277190685272, -0.1179727092385292, -0.4140326678752899, -1.6476514339447021, -0.5416356325149536, 1.5276414155960083, -0.8345929980278015, -0.6004457473754883, -0.7755541801452637, -1.451815128326416, 0.4722329378128052, 0.567638635635376, -1.070480227470398, 1.813754916191101, 1.6332935094833374, 1.4172555208206177, 0.8567987680435181, -0.49394601583480835, -2.5090980529785156, -0.6652393937110901, -0.2480059266090393, -1.0358123779296875, -0.6111456751823425, -0.6470950245857239, -0.10201762616634369, 1.4444483518600464, -0.672861635684967, -0.23701640963554382, 0.24030594527721405, 0.44082626700401306, 0.7561922073364258, 0.17958055436611176, 0.7127069234848022, -0.11194128543138504, 0.6260297298431396, 1.1810334920883179, -0.8731146454811096, -0.8674960136413574, 0.12688185274600983, 0.9817023277282715, -0.9823954105377197, -0.7868607640266418, -2.679769992828369, -0.09011684358119965, -0.6023361682891846, 0.2661738991737366, -0.3147542178630829, -1.280121088027954, 0.12051419168710709, 1.5057034492492676, 0.22640466690063477, 0.22957873344421387, 0.9670140743255615, 1.0059809684753418, 1.2118866443634033, 2.048614740371704, -0.8460274934768677, 0.7135568857192993, -0.4227010905742645, 0.2062426060438156, 1.3398089408874512, 1.2338886260986328, -0.9061678051948547, -0.4093741476535797, 1.5566115379333496, -0.9519486427307129, -1.7489616870880127, -0.04884251952171326, -0.9169225692749023, 0.6077517867088318, -2.112180471420288, -0.9590852856636047, -0.9183914065361023, 0.6427282094955444, -0.1507837027311325, -1.6025843620300293, -0.39379677176475525, 1.1237494945526123, -1.0243048667907715, -1.673568606376648, 1.00197434425354, -0.23563148081302643, -1.2427617311477661, 0.7793315052986145, -0.4431864321231842, 1.0736562013626099, 0.15383945405483246, -0.914364218711853, -0.2467740774154663, -0.7546775341033936, -0.22460703551769257, 0.17180010676383972, 0.15751372277736664, 0.5439450144767761, 0.18752099573612213, 0.5206957459449768, -0.7745699286460876, -0.1063544973731041, -0.30895572900772095, -0.14137281477451324, -1.7399673461914062, 0.5270992517471313, 1.155693531036377, 0.3641619384288788, 1.9550535678863525, -0.5170143842697144, -0.8275490999221802, 1.3165652751922607], [-1.152403712272644, 0.8151581883430481, 2.2702155113220215, 0.5040640830993652, 0.5072062611579895, 0.05300415679812431, 1.1593507528305054, -1.3105182647705078, 0.7339562773704529, 0.8341586589813232, -2.296135187149048, -1.602795124053955, 0.5306637287139893, 1.0731252431869507, -1.1173957586288452, -0.761098325252533, 0.25519782304763794, 0.3290788531303406, -0.9117070436477661, -0.49439048767089844, 1.5570570230484009, 1.3534516096115112, 0.6965647339820862, 1.800652027130127, -2.8205692768096924, -1.3544416427612305, 1.7999122142791748, 1.2586394548416138, 1.7646580934524536, 1.162821888923645, 1.6268655061721802, -0.3980511724948883, -0.3499387800693512, 0.3037344515323639, 1.5467106103897095, 1.567563772201538, 0.6905525326728821, -0.4133780896663666, 0.18535220623016357, 0.6579848527908325, 0.685413658618927, -0.6496420502662659, -1.3098807334899902, 1.1542503833770752, 0.3982464373111725, -0.49970096349716187, 0.47546014189720154, 0.9698224663734436, 0.6673194766044617, 1.8938933610916138, -1.4399532079696655, 1.6478172540664673, -0.23964709043502808, 0.4962087571620941, -0.6638645529747009, 0.6595481038093567, 0.11663303524255753, 0.8910674452781677, 1.3688567876815796, 1.3131736516952515, 0.1401873528957367, 0.519935131072998, -0.9531770348548889, 0.11285708099603653, 0.8036564588546753, 0.46195095777511597, 1.5042427778244019, -1.769251823425293, -1.5707191228866577, -0.03582143038511276, 0.8464320302009583, 1.151005506515503, 1.2367252111434937, -0.6651414036750793, -0.7331194877624512, -0.6316667199134827, 0.6570500135421753, 0.13559190928936005, 0.07582017779350281, -1.1408716440200806, -0.4675625264644623, -1.139032006263733, 1.94923734664917, 0.24742598831653595, 0.7561349868774414, -0.43212002515792847, 0.5874281525611877, 0.6313472390174866, 0.9397153854370117, -0.4488774836063385, 0.01217983104288578, -0.7784990668296814, 0.24697479605674744, 0.26850923895835876, -0.32133176922798157, -0.5081173777580261, 0.4719288945198059, 0.4709736704826355, -0.4388929307460785, 0.26622313261032104, -0.7012346982955933, 0.33571141958236694, 0.6493200063705444, 1.8197665214538574, -0.18640241026878357, 1.7185724973678589, -0.5785400867462158, 0.8462364673614502, 0.49919572472572327, -0.3395836651325226, 1.138243317604065, -0.33491697907447815, -1.5266543626785278, 0.45965084433555603, 0.6562992334365845, 0.43659746646881104, 0.21446143090724945, 0.23092594742774963, 0.1476774662733078, 0.02050331048667431, -0.8000090718269348, -0.578091025352478, 0.17376020550727844, 1.4190800189971924, 1.3356921672821045, -0.055585671216249466, -0.2276393175125122, -0.6676478981971741, -0.8896788358688354, 0.3867633044719696, -1.07573664188385, 0.4724154472351074, -0.7031216025352478, 0.808617889881134, -1.6554707288742065, 1.0899702310562134, -2.3150088787078857, 1.24832284450531, 0.047976478934288025, -0.20645944774150848, 1.267604947090149, -0.17247746884822845, -0.04566739499568939, -0.12666083872318268, 0.23420190811157227, -1.5020782947540283, -1.8244962692260742, -0.154392808675766, 2.292320966720581, -0.1810438185930252, -0.7044768333435059, 0.40351444482803345, 0.306872695684433, -0.19949845969676971, 0.437317818403244, -0.558035671710968, -1.128671407699585, 0.8043456673622131, 0.5510655641555786, 0.0567639023065567, 0.8871300220489502, -0.3515927791595459, -0.06895995885133743, 0.18220534920692444, -1.0841314792633057, 0.07059977948665619, 0.24666476249694824, 0.2557353973388672, 0.21797563135623932, -0.11591242253780365, 0.004736109636723995, -0.27100875973701477, -0.7149989008903503, -1.308983325958252, -0.44762247800827026, 0.4295092225074768, -0.2224481701850891, -0.06561973690986633, -1.0378631353378296, -0.1414034515619278, -0.19199177622795105, -0.8060759902000427, 0.2183675765991211, 1.850001573562622, 0.8323397040367126, -1.430825114250183, 0.5718875527381897, -0.004897750448435545, -0.9095282554626465, -0.7230425477027893, 1.6105189323425293, -0.49801105260849, -1.7057757377624512, 1.3088843822479248, 1.4200694561004639, -1.0243926048278809, -1.3184740543365479, -0.663063645362854, -0.7203363180160522, -0.9048586487770081, -0.4382462501525879, -0.20413678884506226, -0.33276602625846863, -0.45735055208206177, -0.2632511854171753, 1.1783572435379028, -1.3406978845596313, -0.8782580494880676, -0.1856975555419922, -0.6725159883499146, -0.11267206817865372, -0.36292243003845215, -1.5164310932159424, 0.894644558429718, -0.6936174631118774, -0.1572965681552887, -0.4816467761993408, 0.8818633556365967, -1.1062055826187134, -1.7332888841629028, -0.6413941383361816, 0.2408408224582672, 0.06515362858772278, -2.015249490737915, 0.4066927433013916, -1.098631739616394, 1.3959242105484009, -0.5213033556938171, -0.02623223140835762, -1.0977578163146973, -1.1954469680786133, -0.134564608335495, 0.22074536979198456, 0.27040672302246094, -1.839331865310669, 0.15720102190971375, 0.36942294239997864, -1.1906802654266357, 0.584731936454773, -0.8579594492912292, -0.6184592843055725, -0.655971348285675, -1.2882122993469238, -1.3431727886199951, 0.7594410181045532, 0.26073166728019714, -0.6127318143844604, -0.014223771169781685, 0.2423349767923355, -0.38539549708366394, 1.8433629274368286, -1.2894363403320312, 0.6655887365341187, 1.6638977527618408, 0.6961444020271301, 1.3769047260284424, -1.2575465440750122, 0.3265799880027771, -0.4671643376350403, -0.7172537446022034, 0.12383920699357986, 0.33030882477760315, 0.6118143200874329, 1.356748342514038, 0.984299898147583, 1.32415771484375, 0.6568082571029663, 0.33232128620147705, -0.12105634808540344, -0.23630878329277039, -2.5585174560546875, -0.6026036143302917, -0.31237664818763733, 0.4865274429321289, 0.05371977388858795, -0.7760807275772095, -0.24657560884952545, 0.8900243043899536, -1.073706865310669, -0.7782182097434998, -0.8434885740280151, 0.24548909068107605, 1.211050033569336, -0.10987500101327896, 2.2654218673706055, -0.1883772909641266, -1.3064920902252197, 0.5547884106636047, 0.19841106235980988, 0.5447672009468079, -1.390362024307251, -0.7461889982223511, -1.3345739841461182, -0.21295063197612762, 1.0523509979248047, 0.24481995403766632, -1.2530165910720825, -1.6620887517929077, -0.6957269906997681, 1.5593271255493164, 0.8901352286338806, 0.905364453792572, 0.2288169264793396, 0.26477208733558655, -0.4419226348400116, 0.9817619919776917, -1.8995689153671265, 0.9679160118103027, -0.16652795672416687, -0.1265925019979477, -1.7708830833435059, -1.223122477531433, -0.3872438371181488, -0.006041303277015686, 1.4263572692871094, 0.8447774052619934, 0.8173795938491821, 0.07065140455961227, -1.0743049383163452, 0.7855302691459656, -1.4572538137435913, -1.0651261806488037, 0.2935529351234436, -1.183086633682251, 0.25538742542266846, 0.24010302126407623, -1.0783802270889282, 0.9832550883293152, 0.07188400626182556, -1.06111478805542, -0.38918495178222656, -1.0254045724868774, -1.626953125, -0.1172792837023735, -0.3249066472053528, 0.6108536720275879, -0.11143822222948074, -0.4885248839855194, -1.2986048460006714, -1.2743451595306396, 1.3132554292678833, 1.8254207372665405, -0.6117588877677917, -1.3306798934936523, 0.5956398248672485, -0.22086527943611145, 1.0329772233963013, 0.8950237035751343, 0.6460968852043152, 0.9474804401397705, -0.25021785497665405, -1.067158818244934, -0.35023972392082214, -0.16953878104686737, -0.134055495262146, -0.04388546198606491, 0.30194514989852905, -1.9377764463424683, 1.6275705099105835, -0.2535288333892822, -1.6761257648468018, 0.7354609966278076, 0.980084240436554, 0.17959566414356232, 2.4819700717926025, 1.8155012130737305, -0.42091622948646545, 1.0454559326171875, 0.6056418418884277, -2.179985284805298, -0.22067081928253174, 0.9870568513870239, 1.0847631692886353, -0.5411852598190308, -0.41336214542388916, -0.7527207136154175, -0.7521821856498718, 0.35016754269599915, -1.2170745134353638, -0.08511863648891449, -0.8459437489509583, -0.934410810470581, 1.2002043724060059, 1.1367976665496826, 0.22823601961135864, -0.022842423990368843, -0.1270873099565506, 1.2505872249603271, 0.6760548949241638, -1.8831809759140015, -0.6991782188415527, 0.2351546585559845, 0.6499438285827637, -0.04734501987695694, 0.5683201551437378, 1.8575584888458252, 2.317171335220337, 1.0596215724945068, 1.355879545211792, -2.4333176612854004, 1.7124494314193726, -1.9292433261871338, -0.6518900990486145, 1.125641942024231, -1.5440177917480469, 0.1969621330499649, -0.25420430302619934, 2.3965327739715576, -0.8471978306770325, 0.902431070804596, -0.5312137603759766, 0.7500091195106506, 0.37522268295288086, 0.20554080605506897, 0.35300523042678833, -2.324031114578247, -0.6616210341453552, -0.912714958190918, -0.7323567271232605, 0.27943918108940125, -0.8862251043319702, 0.7514362931251526, -0.8394823670387268, -0.222250297665596, 1.341089129447937, 0.347992479801178, 2.2625343799591064, -0.09958403557538986, -2.0372297763824463, -1.7475665807724, 0.7551482319831848, -1.38741135597229, -1.2157244682312012, 1.4536159038543701, 0.08527826517820358, -1.6628093719482422, 1.4788849353790283, 0.6303333640098572, -1.7241774797439575, 2.4541656970977783, 1.388059377670288, -1.0295000076293945, -0.4513378441333771, -1.8915406465530396, -0.7471048831939697, -0.48278728127479553, 1.6241796016693115, -0.23697172105312347, 0.27940380573272705, 0.10004430264234543, -0.10516255348920822, 0.9388890862464905, 0.542076051235199, 0.9891104698181152, 1.461416244506836, -0.3074652850627899, 0.05495411530137062, -1.792650818824768, -0.37656375765800476, -0.10565470159053802, 0.5922698974609375, 2.6904399394989014, 1.7357003688812256, 0.09443453699350357, 0.38747555017471313, -0.956045925617218, -0.8891823887825012, 1.8301876783370972, -0.6702806353569031, -0.2017945498228073, -0.6525911092758179, 1.473555326461792, 0.18887531757354736, -0.024001596495509148, 0.3752758800983429, 0.007503494620323181, -1.2284626960754395, -0.22803635895252228, 0.6561900973320007, 0.6321601271629333, -0.023268112912774086, 0.0008100336417555809, 0.24617943167686462, 0.8892914652824402, -1.1848864555358887, 0.3219889998435974, 1.4173624515533447, -0.6243566274642944, 0.41509002447128296, 1.089787244796753, -0.0059723262675106525, -0.18104304373264313, -0.7209688425064087, 0.3386269509792328, -0.29481440782546997, -0.0697670578956604, 0.9864098429679871, -1.0140684843063354, -0.6258618831634521, -1.5511143207550049, 0.7048346996307373, -0.25218328833580017, -1.399430751800537, 0.005551494657993317, 1.4706891775131226, -0.8963388204574585, -0.03783562406897545, 0.6265888214111328, -0.2535784840583801, -0.8265044093132019, -0.664511501789093, 0.2401917278766632, 0.904487133026123, 0.3361623287200928, 1.581126093864441, -1.0115758180618286, -0.12176281958818436, 0.908871591091156, 0.12942668795585632, 0.8723022937774658, 0.18947827816009521, -0.09203242510557175, 0.09532409906387329, 1.2916338443756104, 0.6941677331924438, 0.6699929237365723, -0.49130070209503174, -1.3079802989959717, -0.32048386335372925, 0.301410436630249, -0.8695900440216064, -2.2535784244537354, -0.05042167007923126, 1.6288686990737915, -0.2317236363887787, 1.4251295328140259, 0.22044789791107178, 0.7408995032310486, 0.023660020902752876, 0.6386829018592834, 0.4406697154045105, 0.30151480436325073, 0.16793984174728394, 1.0361931324005127, -0.3412923514842987, -0.4789259433746338, -1.080446481704712, -0.30063337087631226, 0.4543270468711853, -0.7873555421829224, 0.15387076139450073, -1.8308719396591187, 0.20508094131946564, -0.06275065243244171, -0.1956452876329422, -1.9033604860305786, 0.9090872406959534, -0.7078954577445984, -0.5226040482521057, 0.46224358677864075, -0.5055918097496033, 0.6369127631187439, -0.24794526398181915, -0.02563343197107315, -0.5532512664794922, -1.210576057434082, -0.15723863244056702, 0.3829494118690491, -0.06207382306456566, -0.14507336914539337, -0.804632842540741, 0.5036113858222961, -0.2870882451534271, -0.4781245291233063, 0.42514461278915405, -1.184891939163208, -0.3075607419013977, -0.27559855580329895, -0.5440478324890137, 0.4192831218242645, 1.6124998331069946, -0.46539658308029175, 0.013802528381347656, -0.884942352771759, -0.20891575515270233, 1.1904631853103638, -1.1367464065551758, -0.43705660104751587, -0.3731376528739929, -1.3379496335983276, -0.4873206913471222, -0.7052097320556641, 0.7694600224494934, 1.190078616142273, 0.4579780697822571, 0.8086771965026855, -0.35772705078125, -1.3686119318008423, -1.360461950302124, 0.32712072134017944, -0.7718687057495117, -1.5645660161972046, 0.33042430877685547, 0.27140527963638306, -0.21462948620319366, 0.07684212177991867, -0.8602691888809204, -0.9083225131034851, 0.5813586711883545, 0.2575114369392395, 1.0126152038574219, 0.4953792691230774, 0.26134899258613586, 2.4203431606292725, 0.7102683782577515, 0.14650149643421173, -0.37354010343551636, -1.0480314493179321, 0.27901673316955566, -1.914275884628296, -0.5172591805458069, -0.9865910410881042, 1.3163784742355347, 0.2526853382587433, 0.2501707077026367, -0.2705564498901367, -0.3276773691177368, -0.10165707021951675, -0.6487606167793274, -1.6844853162765503, -0.3948274850845337, -0.6463375091552734, 1.9330086708068848, 1.1416696310043335, -0.6637417674064636, 0.2472420632839203, 1.1324487924575806, -0.2574663758277893, 0.036388181149959564, 0.9477850794792175, -0.46843013167381287, 0.159507617354393, 0.6140400171279907, -0.8630467653274536, 0.32010871171951294, -1.446852445602417, -0.12394165992736816, 0.1231088936328888, -0.5193107724189758, 0.39230474829673767, 0.308998167514801, 0.8682253360748291, 0.8634027242660522, -0.7704665064811707, -0.7840114235877991, -0.786970853805542, 1.6382911205291748, -1.7389447689056396, -0.515011727809906, -0.7489766478538513, 1.3734785318374634, -1.2256120443344116, 0.37204039096832275, -0.47476908564567566, 0.007034550420939922, 1.7356442213058472, -0.11372225731611252, -0.11385476589202881, 1.6613894701004028, 0.8183196187019348, -0.14279551804065704, -0.5988458395004272, -0.4753098785877228, -1.0450725555419922, 2.455491781234741, -1.1092137098312378, 0.142498180270195, -1.9783000946044922, 0.5238518714904785, -0.6039547920227051, 1.0130664110183716, -0.06280159950256348, 0.7894381284713745, -1.4535702466964722, -0.009234132245182991, 0.2674824297428131, -0.9308967590332031, 0.5339004993438721, -0.20303170382976532, -0.3075674772262573, 1.7090508937835693, 0.4934186637401581, 0.6324573159217834, 0.4511924982070923, -0.525658369064331, -2.276470184326172, -1.180749535560608, 0.8431166410446167, 0.5532636642456055, 0.20308460295200348, 1.5371321439743042, 0.3487953245639801, 0.572978138923645, -1.1130738258361816, 0.041183944791555405, 0.5834426879882812, 0.16920112073421478, -0.5827708840370178, 0.8480435013771057, 0.174138605594635, 0.5448704361915588, 0.09481361508369446, -0.09953011572360992, 1.3623738288879395, 0.8744859099388123, -0.138626828789711, -1.438738226890564, -0.3490183353424072, 0.02790691889822483, -1.6898318529129028, 0.9117270708084106, 0.14033523201942444, -0.21714277565479279, 0.7756335735321045, 1.5193698406219482, 0.07128441333770752, 1.1849406957626343, 1.7366589307785034, -0.7127872705459595, 1.5428491830825806, -0.013999643735587597, 0.45051777362823486, 1.7529352903366089, -1.3985254764556885, 0.140761598944664, -0.19363127648830414, -0.10906728357076645, -0.5559423565864563, -0.15853498876094818, -0.9247734546661377, 0.1911163330078125, -0.15531228482723236, 1.2404378652572632, 0.0854296013712883, -0.42544642090797424, 0.6142910718917847, -0.21353331208229065, 0.3467044234275818, -0.260001003742218, 0.0857616662979126, 0.8432316780090332, -0.4743589162826538, -0.8518418073654175, 0.2436458021402359, -1.716467261314392, -0.9657207727432251, 0.14534161984920502, 0.247883141040802, -0.7601746320724487, -1.9143319129943848, 1.3369007110595703, -0.4992631673812866, 0.38446924090385437, 0.4771302342414856, 0.06393697112798691, 1.1928952932357788, 0.19639331102371216, -1.4551284313201904, -0.9249463081359863, 1.2140872478485107, -0.10238221287727356, 1.3448072671890259, 1.4048576354980469, -1.167601466178894, 0.4568996727466583, -0.21062920987606049, -0.15601809322834015, 0.07731861621141434, -0.22171366214752197, 0.025388624519109726, -1.4471166133880615, -0.548378050327301, 2.663721799850464, -0.4565795660018921, -1.123260736465454, 0.3925020098686218, 0.8396424055099487, 1.7028520107269287, 0.06115810573101044, -1.207214593887329, 0.6078508496284485, -0.8483351469039917, 1.6932892799377441, 0.15021023154258728, 0.6611741781234741, 0.32092639803886414, 0.20813995599746704, -0.7577598690986633, 0.5969309210777283, 1.0247163772583008, 0.27290838956832886, 0.3554176390171051, -1.326278805732727, -0.17937712371349335, -1.921187162399292, -0.2764490246772766, 1.2273403406143188, -1.8404340744018555, -0.043724313378334045, 0.2892483174800873, 0.2650510370731354, -0.13629314303398132, -1.0448060035705566, -1.4938780069351196, -0.5840694308280945, -1.526983380317688, 0.8638118505477905, 0.19259631633758545, 1.5591403245925903, -1.2755461931228638, 0.42038777470588684, 0.13748225569725037, 1.0853341817855835, 0.32537513971328735, -0.7778801321983337, 0.855848491191864, -1.0298184156417847, -0.8528520464897156, -0.7143529057502747, -0.8880763649940491, 0.17355085909366608, 0.3165409564971924, 0.4084651470184326, -1.5962910652160645, 0.051593683660030365, -0.5602469444274902, 1.339937448501587, 0.9725989699363708, 0.6567733883857727, 1.7066096067428589, -1.9112136363983154, 1.8175735473632812, 1.0227669477462769, 0.37846487760543823, -1.1236321926116943, -1.046849250793457, -1.4573200941085815, -0.43281036615371704, 1.398524284362793, -0.8628460168838501, -0.6970614790916443, 3.0243537425994873, -1.4677144289016724, 0.5194341540336609, 0.03631926327943802, -1.7434589862823486, 1.6701886653900146, -2.384697437286377, 1.479502558708191, -0.01164119690656662, 0.11535812169313431, -1.1957885026931763, 0.947006106376648, -0.14033515751361847, 0.16090326011180878, 0.37054312229156494, 1.5563358068466187, 1.4739145040512085, -1.0735111236572266, -0.7648054957389832, 1.5228533744812012, -1.2602479457855225, 0.0011141410795971751, -0.17493222653865814, -0.7549651265144348, -1.002569556236267, -1.449846863746643, 1.067075252532959, -0.4065994322299957, 0.4601084887981415, -1.5060127973556519, -0.5465350151062012, -0.25382500886917114, 0.5087568759918213, -0.3175726532936096, -1.5562454462051392, -1.4297418594360352, -0.053561944514513016, 1.8259813785552979, -2.316723585128784, 0.1185862272977829, 0.6662863492965698, -1.0167101621627808, 1.1671584844589233, -0.4433729946613312, 0.1270516961812973, 0.10414818674325943, 0.019687950611114502, -2.058821201324463, 0.4721367359161377, 0.6658205986022949, -0.6607251763343811, 1.3818937540054321, 1.3985975980758667, 0.17346899211406708, 0.20602211356163025, -0.756430983543396, -0.2574083209037781, 0.5249790549278259, 1.5935457944869995, 0.5612732768058777, 1.014109492301941, 0.2264656275510788, -0.3238776624202728, -1.8299943208694458, 2.345331907272339, -0.25951263308525085, 0.6916019320487976, 0.6383712291717529, 0.18414798378944397, -1.9811071157455444, -0.4248693585395813, -0.8359118700027466, 1.1623966693878174, -0.23948955535888672, 1.3051995038986206, -0.403250128030777, 0.13993728160858154, 0.6013507843017578, -0.2962604761123657, 0.25200513005256653, -2.581895589828491, 1.4728140830993652, -0.3283664286136627, -0.7239342927932739, 2.252593994140625, -0.8586204648017883, 0.559995174407959, -1.0745632648468018, -1.4022523164749146, 0.25106188654899597, -0.6642249226570129, 0.3465861976146698, -0.10673481225967407, -0.36922743916511536, -1.7930635213851929, 0.5894050002098083, 0.1092195063829422, -0.7691522240638733, -0.491056889295578, 0.25359973311424255, -0.5088434219360352, -1.0405384302139282, -1.7166279554367065, 0.4450835883617401, 0.5316945910453796, 0.6756975054740906, 0.3605336844921112, 0.7969736456871033, 1.6747506856918335, -0.6506595015525818, 1.0618714094161987, -1.4904450178146362, -0.8133246302604675, -0.6428374648094177, 0.46872639656066895, -0.10748376697301865, -0.062040239572525024, 0.4467073082923889, -0.5644960403442383, -0.27380087971687317, -0.05546172708272934, -0.8451697826385498, 2.0368847846984863, 0.718984067440033, 0.05473243445158005, -0.875937283039093, 0.8506674766540527, -1.4849698543548584, 0.07241823524236679, -0.7810099124908447, -2.9877185821533203, -1.103937029838562, -0.3431652784347534, 0.5220388770103455, -1.7437690496444702, 1.311048984527588, 0.1694425493478775, -1.671380639076233, -0.02699972689151764, 0.45413264632225037, 0.027909234166145325, -0.580642580986023, 3.1607000827789307, 0.19001229107379913, 0.8349326252937317, -2.806746482849121, 1.601949691772461, -0.34654587507247925, -1.2513303756713867, -0.6883057355880737, -0.6678145527839661, -0.7794039845466614, -0.4548581838607788, 0.1882099211215973, -0.9059205055236816, 1.369625210762024, 0.8004751801490784, 0.10778842866420746, -0.9865517020225525, 0.0582423098385334, -1.7048163414001465, -0.25856122374534607, -0.7223221063613892, 0.07389062643051147, -0.6260473728179932, 0.07644455879926682, -0.7785981893539429, 0.29517972469329834, -0.42978811264038086, -0.1384536623954773, 0.09872329980134964, -0.055330146104097366, -0.9847782850265503], [-1.186562418937683, 1.5499335527420044, 0.9959216713905334, 1.0611268281936646, -1.7564489841461182, -0.8438887596130371, 0.515466034412384, 1.1411396265029907, 1.4142204523086548, -2.439056634902954, -1.1020702123641968, -0.273593008518219, 1.0162853002548218, -0.4284299612045288, 1.2138944864273071, 0.18026386201381683, -1.2940661907196045, 0.7413860559463501, -0.6597288846969604, 0.898862361907959, -0.11448105424642563, -1.644982099533081, -0.6613712906837463, -0.9731057286262512, -0.039103761315345764, -0.17608442902565002, -0.05267778038978577, 0.13951829075813293, -2.530508279800415, 1.295438289642334, -0.5227952599525452, -1.1228845119476318, 0.05972226709127426, -0.2496674507856369, -0.41071078181266785, 0.4679218530654907, -0.5261879563331604, -0.11041650176048279, -0.25189322233200073, 0.5368248224258423, -0.5010400414466858, 1.2533403635025024, -0.6487688422203064, -1.2782195806503296, -0.6883775591850281, 1.0113658905029297, -1.568826675415039, 0.8084359765052795, 1.0598770380020142, -0.35291269421577454, -1.2929922342300415, 0.2651076018810272, 0.8309959769248962, -0.6182177066802979, 0.3576143682003021, -0.9974488019943237, -1.7056061029434204, 1.269423007965088, -0.37182119488716125, 0.5757709741592407, -0.14053121209144592, 0.40865615010261536, 1.5114612579345703, -0.5433206558227539, 1.625841498374939, -0.25273430347442627, -0.0939585417509079, -1.404158115386963, 0.015039741061627865, -0.41225239634513855, 1.4714230298995972, 1.876354455947876, -0.18931275606155396, -1.6204767227172852, 0.6998358964920044, -0.9792661666870117, 1.6060328483581543, 0.20033542811870575, -0.8675810098648071, 0.3065125048160553, 1.4795714616775513, 1.0686795711517334, -0.1092662513256073, 0.5347045063972473, 1.627730369567871, -0.0683131143450737, 0.2188110500574112, 1.2020902633666992, -1.751510500907898, 1.1353050470352173, 1.2839977741241455, 0.054411716759204865, -2.467006206512451, -1.5161209106445312, -0.5818216800689697, 1.1158555746078491, 0.14820004999637604, -1.5164272785186768, 0.9699624180793762, 0.4674714505672455, -1.9602264165878296, -0.06092338636517525, 0.6456030011177063, -0.27701759338378906, 0.15987880527973175, -1.8699815273284912, 0.7353333234786987, 1.0596603155136108, 0.4992857873439789, -0.2558128237724304, 0.47081080079078674, -0.6409976482391357, 0.36643144488334656, 0.6642631888389587, 0.7995613813400269, 0.25902071595191956, -0.2735992670059204, -0.580422043800354, 0.6242237091064453, 1.3421046733856201, -0.5030606985092163, -1.9870554208755493, 0.7534832954406738, 0.20218604803085327, -0.18481291830539703, 0.6554688215255737, -0.5998948812484741, 0.24960973858833313, 1.395384669303894, -1.0104573965072632, -1.5142101049423218, 0.8674442768096924, 0.8820257186889648, 2.618483304977417, -0.1285001039505005, -0.5106443166732788, -0.8255165815353394, -1.981196403503418, -0.7244351506233215, -0.3547094464302063, 1.0727864503860474, 0.4384392201900482, 0.4628627300262451, -1.4681452512741089, 0.32135817408561707, 0.057127658277750015, -1.1035270690917969, 0.521649181842804, 1.2230703830718994, 0.6234549880027771, -1.1053264141082764, -0.3325081467628479, -0.4041607975959778, 0.7033506631851196, -1.193102478981018, 0.2933478355407715, 0.42042985558509827, 0.9433244466781616, 0.11157229542732239, 0.5160189867019653, -0.3542470633983612, -0.9310851097106934, -0.2542528510093689, 0.48377037048339844, -0.8600428104400635, 1.887886643409729, -0.3045949339866638, 0.01833471842110157, 1.5787357091903687, -0.05713411793112755, 0.07413949072360992, 1.8429256677627563, -0.18543271720409393, -0.40937474370002747, 0.3658286929130554, -0.7642459869384766, -0.9719566702842712, -0.33538487553596497, -0.10236728936433792, -2.1526083946228027, -1.104443073272705, 1.1544840335845947, -1.555848240852356, -0.02670198678970337, -0.1347779780626297, 0.201712504029274, 1.167707085609436, 0.048166368156671524, 0.13433131575584412, -0.605000913143158, 0.939835250377655, -0.5562577247619629, -0.8640298843383789, -0.17452162504196167, 0.29617375135421753, -2.0774166584014893, -1.8087059259414673, -2.5628464221954346, -2.1462910175323486, 0.7559977173805237, -0.0066127777099609375, -1.3904063701629639, 0.9090027809143066, 0.5424004197120667, 0.024439306929707527, 1.5444811582565308, -0.3696897327899933, -0.18621449172496796, -0.2478329837322235, -0.594596803188324, 1.0671634674072266, -1.435781478881836, 0.8876633048057556, -0.27967819571495056, 0.25592201948165894, -0.16519486904144287, 0.8456778526306152, 1.2849338054656982, -0.5836033821105957, 1.4430625438690186, 2.1523447036743164, 0.5060873031616211, -0.7123037576675415, -1.8938809633255005, 0.8873408436775208, -1.1639641523361206, -0.9227046966552734, -0.429340124130249, -1.1308766603469849, 0.0015811464982107282, -1.6687735319137573, 0.747342050075531, 0.006263106130063534, -1.6281570196151733, 0.7353727221488953, 0.24457626044750214, -0.18793262541294098, -0.4627027213573456, 1.5156038999557495, -0.04548304155468941, 1.183367133140564, 0.5835714936256409, 2.6843581199645996, -0.532080352306366, -0.8507782220840454, 1.5094783306121826, -0.7589766383171082, 0.395244300365448, 0.35049334168434143, -0.36340588331222534, -0.5608193874359131, 0.12211146205663681, -0.146025612950325, 1.2907907962799072, 0.7446668744087219, 0.3651903569698334, 0.24706174433231354, -2.2732062339782715, -0.6142061948776245, -1.330818772315979, 0.24720685184001923, -0.7113295793533325, -0.3348691165447235, -1.445946455001831, 0.19143761694431305, -0.8760602474212646, -0.23574034869670868, -0.5482670664787292, 0.9169753789901733, 1.3159589767456055, 1.069995403289795, -0.8430453538894653, 0.6996311545372009, 1.9709521532058716, -0.2688227593898773, 0.6744199395179749, 0.09857026487588882, -0.5301608443260193, 1.0060054063796997, -1.4483152627944946, 1.411899209022522, -0.022225271910429, 0.8342833518981934, 0.16890691220760345, 0.1530526727437973, -0.38576099276542664, 1.538992166519165, -0.4809299111366272, 0.31527456641197205, 0.045943912118673325, 0.5932703614234924, -0.18834927678108215, 1.9479533433914185, -0.9473446011543274, -1.2604427337646484, 0.27774307131767273, -1.3005164861679077, -0.16866998374462128, -2.6566202640533447, -0.6906628012657166, -2.0062105655670166, 1.1176564693450928, -1.022316336631775, 0.5557026863098145, 0.6557980179786682, -1.111004114151001, 0.32994163036346436, -1.6575052738189697, -0.036417290568351746, 0.7815825939178467, 1.4859979152679443, -0.9282599687576294, -0.30143067240715027, -0.65352863073349, 0.5666095018386841, -0.4912947118282318, 1.503869891166687, -0.06206129863858223, 0.9073962569236755, -0.6883004903793335, 0.1271725296974182, 2.4375789165496826, -0.2764861285686493, 1.6683485507965088, -1.3962805271148682, -0.48571979999542236, -0.2622910737991333, -0.5934770703315735, 0.4579082131385803, 0.5537233352661133, 0.16100828349590302, -0.13509131968021393, 0.24861353635787964, -0.05316845327615738, 2.3405370712280273, -1.7959803342819214, -1.3962520360946655, -2.485330581665039, -0.0416378490626812, -1.5608940124511719, -1.6470282077789307, 0.42869651317596436, 2.2670986652374268, 0.376017689704895, 1.566818356513977, 0.21990741789340973, -0.3122864067554474, 1.326770544052124, -0.032830700278282166, -0.18905261158943176, 1.5132499933242798, 0.5556344389915466, -0.6363429427146912, 0.29896506667137146, -0.3742823898792267, -0.2177131175994873, 2.5344502925872803, 0.3365727961063385, -1.0313808917999268, -0.11795621365308762, 0.7602633833885193, 0.4540208578109741, -1.1007012128829956, 0.9016944169998169, -1.58596932888031, 1.0767338275909424, -0.3763982653617859, -1.4675610065460205, -0.8552945852279663, 0.5441396236419678, 0.34540486335754395, 0.5930382609367371, 0.3449043035507202, -0.4849967360496521, -1.9160737991333008, -1.7004433870315552, -0.4661877155303955, 0.29973891377449036, 0.1883644461631775, -0.200677752494812, -1.3487772941589355, -0.12764838337898254, -0.2139563262462616, -1.184748649597168, 0.17856933176517487, 1.725988507270813, -0.09638737142086029, 1.6356911659240723, -0.4504855275154114, -0.4221235513687134, 0.5542755126953125, -1.5746140480041504, 0.09928932785987854, 1.256316065788269, -0.36255139112472534, -0.6443589329719543, 0.16187657415866852, 1.3444952964782715, -0.9962087869644165, 0.38182568550109863, 0.3496791124343872, -0.12974411249160767, -0.6513646245002747, 1.0249953269958496, -1.144484281539917, -0.34810078144073486, -0.5918768644332886, -0.9400498270988464, -0.11425180733203888, -1.6506959199905396, 1.2429953813552856, 0.08727807551622391, 0.047903310507535934, 0.9244593977928162, -0.19071520864963531, -1.8051557540893555, -0.30278095602989197, -0.5306181907653809, -1.8508844375610352, -0.6368858218193054, 1.1747380495071411, 0.4373951256275177, 1.1632957458496094, -1.1713718175888062, 0.37405338883399963, 0.3177766501903534, 1.9335168600082397, 0.34918707609176636, -1.0527880191802979, -0.14892585575580597, -0.825657069683075, -0.18492506444454193, 0.835252046585083, -0.5548198819160461, 0.6681808829307556, -0.30910786986351013, 0.3740729093551636, -0.19414472579956055, 2.2309868335723877, 0.0812835842370987, 1.150234341621399, -2.5447263717651367, -0.32970356941223145, -0.4394080340862274, -0.8144022226333618, 0.7393830418586731, 0.9554659724235535, -1.5881367921829224, 0.1803104430437088, 0.21836504340171814, 0.20620539784431458, 1.788706660270691, -1.5674376487731934, -0.9788743853569031, -1.2482703924179077, 0.9119641780853271, -1.2635157108306885, 1.1455174684524536, -1.8676772117614746, 1.692082166671753, -1.212838888168335, 1.1536799669265747, 0.0068465727381408215, -1.8088510036468506, -0.748302161693573, -0.4234658181667328, 0.4795188009738922, 1.048093557357788, -0.0383654348552227, 1.1107597351074219, 0.6538166403770447, -0.7637084126472473, -0.3815816342830658, -0.8147712349891663, 0.5707111954689026, 0.022905809804797173, 1.0922547578811646, -0.05527140200138092, -1.4372172355651855, -0.35997605323791504, -0.9454636573791504, 0.8470413684844971, 0.7376500368118286, -1.3239655494689941, -0.05115264281630516, -0.0038068799767643213, -0.6754588484764099, 0.6826633214950562, 3.0740246772766113, -0.5443260669708252, -1.675833821296692, -0.41938820481300354, 1.6246737241744995, 0.05965230241417885, 2.3614511489868164, -0.12742988765239716, -2.209091901779175, 0.05137992277741432, -1.0647892951965332, 1.4587949514389038, -1.7775429487228394, 0.030243920162320137, -0.2462516725063324, -1.508053183555603, 0.5199921131134033, -0.15494711697101593, -0.1299719661474228, 0.32688605785369873, -1.283382773399353, 0.48631253838539124, -0.5768518447875977, 1.0817663669586182, 1.4239410161972046, 0.37900272011756897, 0.9836615324020386, 0.010358063504099846, -0.11261944472789764, 1.1528774499893188, -1.536535382270813, -1.9836552143096924, 0.41774824261665344, 0.25814276933670044, -2.0672202110290527, 0.12509199976921082, 0.7384559512138367, 0.2911549210548401, 0.18784217536449432, -0.9374586343765259, 0.2705047130584717, -0.47926226258277893, 0.1372895985841751, 2.5673465728759766, -1.797420620918274, -1.6387951374053955, -2.418001174926758, 1.6551438570022583, -0.8108844757080078, 0.948185920715332, -0.2630717158317566, 0.3461345136165619, -0.25164100527763367, -0.9177313446998596, 2.0204360485076904, 0.12611347436904907, -1.0260920524597168, -0.4351004660129547, 0.103507399559021, -0.2867411673069, -0.24611122906208038, -0.2508411407470703, 1.363761305809021, -0.8255912661552429, 0.20298543572425842, -0.9288275837898254, 0.6273216009140015, -0.971063494682312, -0.10774653404951096, 0.2003040462732315, -1.3606864213943481, -0.913878321647644, -0.5855690240859985, -0.8388387560844421, 1.0431396961212158, -0.9787137508392334, 0.3728232979774475, 1.0908790826797485, -0.3686097264289856, 1.522557020187378, -0.777401864528656, 0.14662711322307587, 0.32671990990638733, -0.048213642090559006, 0.5618823766708374, 1.1716103553771973, -0.8026050329208374, 0.7185812592506409, -1.62567937374115, 0.2767883241176605, 1.1056485176086426, 0.2141164392232895, 2.321470260620117, 1.2607920169830322, -0.1389489620923996, 1.8255783319473267, -1.5208240747451782, 1.0823521614074707, 0.9314284324645996, 0.5430726408958435, 0.8584297299385071, 0.33958572149276733, 0.41803303360939026, -1.3698571920394897, -1.7543751001358032, 1.3775476217269897, 0.9648281335830688, -0.3065018057823181, -0.5031322836875916, -0.4089650511741638, -1.5522083044052124, 1.2899175882339478, -0.7462807297706604, 2.467773199081421, -0.45882442593574524, -0.20157213509082794, -1.2496963739395142, 0.9716576933860779, 0.07982382923364639, -0.617402970790863, -1.1396381855010986, 0.07237692922353745, 0.09651920199394226, 0.012109906412661076, -1.4176701307296753, -0.9102378487586975, -0.6869024038314819, -0.3172725737094879, 1.1417144536972046, 0.461752325296402, 0.7276037335395813, 0.7547703385353088, 0.43793055415153503, -0.049242839217185974, 0.40446358919143677, 0.5570394396781921, 0.9992570877075195, -1.3293852806091309, -0.09734448045492172, 0.5248344540596008, 1.4378741979599, 1.8452092409133911, -1.9580574035644531, -2.151256561279297, -1.1576210260391235, 0.4703817069530487, 1.3604888916015625, 2.0175631046295166, -0.6570963859558105, -0.30179664492607117, 0.8290688395500183, 2.9307336807250977, -0.2641837000846863, 0.2376454472541809, -0.622346818447113, -0.7652646899223328, 0.959405779838562, 1.6525399684906006, -0.000526759191416204, -0.604270875453949, 1.4677746295928955, -0.8360610008239746, -0.37590864300727844, -0.36812669038772583, 0.8727656006813049, -1.357957363128662, -0.5574522614479065, -1.3416568040847778, -0.15922598540782928, 0.5212719440460205, 0.6543868780136108, 0.42578133940696716, 1.4803504943847656, -1.284381628036499, -1.246827483177185, 1.3523895740509033, -0.07000832259654999, 0.6990017890930176, 0.197310209274292, 1.223833680152893, -0.15233062207698822, -0.8298113942146301, -0.2879572808742523, -1.8866157531738281, -0.7650480270385742, -1.624511957168579, 0.3699088990688324, 0.30703407526016235, 1.5262718200683594, -1.1331886053085327, -0.426326185464859, 0.30053797364234924, -0.16985660791397095, -1.8966237306594849, -0.5814198851585388, -0.8308656215667725, -1.2674765586853027, 2.046098470687866, 0.9199355244636536, 0.5903276205062866, 1.047978401184082, 2.167741537094116, -0.1856759637594223, 0.7521551251411438, 0.6169095039367676, 1.50233793258667, -0.05382451415061951, -0.4994160532951355, 1.3673311471939087, -0.0682557076215744, -0.2524447739124298, -0.5378397107124329, -1.2183215618133545, -0.4994351267814636, -0.10568542778491974, -0.3954814076423645, -0.499171644449234, 0.9837685823440552, -0.5491440296173096, -0.14774103462696075, -0.1147039607167244, -0.3649148643016815, 1.4570034742355347, -0.6252261400222778, 0.4795342683792114, -1.445005178451538, -0.6367781162261963, 0.07280942797660828, 0.9534331560134888, -0.2879812717437744, -0.009481974877417088, -0.731106698513031, -1.2457627058029175, -0.75473952293396, -1.9650870561599731, 0.07829456031322479, -0.5473757982254028, -1.5345888137817383, 0.3109642565250397, 0.08861034363508224, -0.13133493065834045, -0.1850298047065735, 1.571303129196167, 0.008625864051282406, -0.7968623638153076, -0.1630595177412033, -1.4102219343185425, -0.16874323785305023, -0.4462222158908844, 0.3915478587150574, 0.2229248434305191, -0.08943232148885727, -3.2047770023345947, -1.4019736051559448, 0.13561390340328217, -0.22008775174617767, 1.3625305891036987, -0.8985477089881897, -0.5903275609016418, 1.4984701871871948, 1.80925714969635, 0.4108603596687317, 0.8570065498352051, -1.5135109424591064, -0.19091090559959412, 0.011184810660779476, -0.0025325955357402563, 1.5277557373046875, -0.30761778354644775, -0.6931808590888977, -0.8313575983047485, -1.6553581953048706, -0.9064152836799622, 0.5711187124252319, 0.6975020170211792, 0.37489184737205505, 0.9613269567489624, 1.4876333475112915, -1.2533162832260132, 0.9598422646522522, 0.5075413584709167, -0.3989872634410858, -0.035062525421381, -1.6659499406814575, 0.09020194411277771, 0.46565914154052734, 0.4050367772579193, 1.1906565427780151, 1.5805913209915161, 1.3610378503799438, 0.23982441425323486, 0.5349392294883728, -1.2402291297912598, 0.5302863717079163, -0.21273347735404968, 0.47685354948043823, 1.5939955711364746, -0.30593422055244446, 1.4868146181106567, -0.27673110365867615, 0.24151542782783508, 0.2838881313800812, 0.8337145447731018, -1.1240811347961426, 1.2504044771194458, -0.06587865948677063, 0.8739574551582336, -0.7692108750343323, -0.3715885579586029, -0.28281447291374207, -0.6154981851577759, -0.1348336786031723, -0.7512627840042114, 0.9135497212409973, 0.4922533333301544, 0.42458194494247437, -0.8913847208023071, -1.1455109119415283, 1.7180215120315552, -1.4100441932678223, 0.3440250754356384, -0.024535756558179855, 0.11057016253471375, -0.43017950654029846, 0.3782974183559418, -1.17476224899292, 1.3673204183578491, 0.9810916781425476, 0.30330273509025574, -0.2843931317329407, 0.3960839807987213, 2.23786997795105, 1.2051644325256348, 1.2109355926513672, -0.7438621520996094, -0.16445156931877136, 0.16817639768123627, 0.8333378434181213, 2.578310251235962, 1.1803228855133057, 0.6333318948745728, -0.7249866127967834, -0.16240887343883514, 1.37015700340271, -2.3353471755981445, -0.26671236753463745, 3.765909194946289, -0.06074392423033714, -0.4469359517097473, -0.0871611163020134, -0.36338427662849426, 0.8338046669960022, -0.4223313331604004, 0.8009139895439148, 1.0285753011703491, -0.7874038219451904, 0.5764341354370117, -0.8222293853759766, -0.810636579990387, 0.8041428327560425, -1.532441258430481, -1.1642452478408813, 0.11078977584838867, -0.1740153431892395, -0.8394407033920288, 0.5873681306838989, 0.6296730637550354, 1.493779182434082, -0.38535135984420776, -1.2750884294509888, 0.6450014710426331, -0.34279948472976685, -0.3936690390110016, 0.04186512157320976, -0.6594474911689758, 0.04540897160768509, 1.2080703973770142, -0.9062883257865906, 0.5047386884689331, -0.10293793678283691, 0.23390963673591614, -1.7001186609268188, -0.6126470565795898, 2.4486052989959717, 0.23588912189006805, 0.9810081720352173, -0.10802026093006134, -1.1383334398269653, 0.3761507272720337, -1.6056801080703735, 0.8312590718269348, 2.6950902938842773, -2.4745116233825684, 0.1119813546538353, -0.41550421714782715, 0.031276702880859375, -0.320007860660553, 0.09117575734853745, 0.007386391516774893, -0.4173462688922882, -0.028173720464110374, -1.4737539291381836, -1.0867305994033813, 0.07100886106491089, -1.8555163145065308, 0.183378204703331, -0.07921325415372849, 1.8869761228561401, -0.41041100025177, 0.8130943775177002, 0.5662814378738403, 1.6476818323135376, -1.338181495666504, 0.38624683022499084, 1.6853957176208496, -2.6551530361175537, 2.197711229324341, -0.6207361817359924, -0.8523847460746765, -1.523004174232483, -1.4454327821731567, -0.1848112940788269, -0.2649061679840088, 0.31947141885757446, 1.0567247867584229, -0.12232034653425217, -1.7570143938064575, -0.6807119846343994, -1.5276168584823608, -1.366352915763855, -0.697171151638031, -1.1964811086654663, 1.01958429813385, -0.8238013386726379, -1.6073957681655884, -1.5155274868011475, 0.3755432069301605, -0.5946248173713684, -1.994983196258545, -0.43682608008384705, 0.0782143622636795, 0.21598654985427856, -1.8014521598815918, 0.8105756044387817, 0.5484070181846619, 1.6535725593566895, -0.3720681667327881, -0.7743398547172546, 0.6039810180664062, 0.08550399541854858, 1.1865355968475342, 1.1787316799163818, -0.564106285572052, 0.20575912296772003, 0.9051389098167419, -1.7912119626998901, 0.41811108589172363, -0.485558420419693, 0.24346347153186798, 1.2607717514038086, 0.2661277651786804, -0.8285678625106812, -0.38898995518684387, -0.5283781886100769, -0.83447265625, 0.837222158908844, 0.47158679366111755, 0.24762766063213348, 0.6108036637306213, -0.16676224768161774, 1.4056885242462158, 0.8319390416145325, -0.28648895025253296, -1.1871041059494019, -0.29680922627449036, -1.1643545627593994, 0.9072238206863403, -0.3544327914714813, 1.446984052658081, 0.7344962954521179, -2.558751344680786, 1.028557538986206, -1.0142611265182495, 0.058401353657245636, 0.9261752963066101, -0.21653416752815247, 0.8939303159713745, -1.2344101667404175, -0.2636868953704834, -1.3504109382629395, -0.811389684677124, -1.017730474472046, -0.9247062802314758, -1.7590389251708984, 0.6002658605575562, -0.8546078205108643, -0.8863921761512756, 0.6467087268829346, -0.7944694757461548, -0.470522403717041, 1.4936330318450928, -1.3881092071533203, -2.0850977897644043, 0.16802673041820526, -0.8919510245323181, -0.9780735969543457, -0.7866247296333313, -0.37435978651046753, 1.5505906343460083, -0.18372447788715363, -0.48451703786849976, 0.5685681700706482, -0.45555272698402405, 1.6402757167816162, 1.7582956552505493, 0.6604264378547668, 0.4369266629219055, 1.0547515153884888, 0.14326375722885132, 0.6471478939056396, 1.1839133501052856, -0.464297354221344, -1.4348633289337158, -0.1519765704870224, -0.9276345372200012, 0.5783218741416931, -1.345196008682251, 0.5300531387329102, -0.6785220503807068, 1.4011988639831543, -0.5871100425720215, -0.43178439140319824, -0.47957175970077515, -0.9321155548095703, -0.5524045825004578, 0.4641227722167969], [1.472076416015625, -0.3765624761581421, -0.930657684803009, 0.27723655104637146, 0.970105767250061, -1.225694179534912, 0.03703998774290085, 0.7666406035423279, 1.6371501684188843, -2.4230923652648926, -0.5945367217063904, 1.7473342418670654, 0.5781386494636536, 0.12195733934640884, -0.9369815587997437, 0.605029284954071, 0.16732360422611237, 0.6269717812538147, -1.4243971109390259, 1.4400392770767212, 0.03793832287192345, 1.3725208044052124, -0.2854701578617096, 0.40365496277809143, -1.1856837272644043, 0.891616702079773, 0.686600923538208, 0.4032765030860901, 0.21248604357242584, -0.10718502849340439, -3.341722011566162, -0.8209835290908813, -1.2829734086990356, 2.386932373046875, 0.3762650787830353, -1.0410863161087036, -0.7612301707267761, -0.6289650201797485, 0.702408492565155, 0.009675323963165283, -0.6359764337539673, 1.1625293493270874, -0.200348898768425, 0.1942378729581833, 0.0735933855175972, -0.4932819604873657, 0.6885384917259216, -0.5252489447593689, -1.9799574613571167, 0.3416009545326233, 0.22853055596351624, 0.1239568293094635, -0.7719748616218567, -0.619757354259491, -1.02986741065979, 0.03547089919447899, 1.0838733911514282, 0.30134961009025574, -1.2712377309799194, -0.4490729868412018, -0.1286274492740631, 0.22251778841018677, -1.2755887508392334, 2.3558921813964844, -0.32968586683273315, 0.3846464455127716, 0.9861904978752136, -0.3315698802471161, 0.7722933888435364, 0.9893977046012878, 0.8362823724746704, 0.19987472891807556, 0.05025133490562439, 0.8771190643310547, 1.078548789024353, 0.913459062576294, 0.704049289226532, -1.7007174491882324, 1.9807997941970825, -2.0061068534851074, -1.1881462335586548, 0.3730260729789734, -0.9494807720184326, 0.743394672870636, 2.5079782009124756, -0.28446587920188904, -0.061727359890937805, 0.6212741136550903, -0.40056559443473816, 0.9849675297737122, -2.3382411003112793, 0.8761559128761292, -0.643977701663971, -0.913827657699585, 0.14161966741085052, -0.9240137338638306, -0.2748439610004425, 0.7792974710464478, 1.6589432954788208, -0.7458868026733398, -0.2162690907716751, -0.8470495343208313, -0.42770418524742126, 1.1627143621444702, 0.4563184082508087, 0.949306845664978, -1.5128883123397827, 0.900933027267456, 0.4338894188404083, 2.0901801586151123, -0.21769925951957703, 0.2539432644844055, -1.0909442901611328, 0.5507603287696838, -0.8276297450065613, -0.5682231187820435, -2.875136375427246, -0.2474553883075714, -0.10714415460824966, -0.7762535214424133, -1.6382887363433838, 0.166802316904068, 1.4217830896377563, 0.8052727580070496, 1.1469736099243164, -0.6991814374923706, 0.0020016233902424574, -1.5419198274612427, -0.8519207239151001, -0.2778896689414978, 0.6358559131622314, 0.4631343185901642, -0.3508346974849701, 0.9668034315109253, 0.6899840235710144, -0.9758484363555908, 0.7944196462631226, 1.340255618095398, -0.0766618549823761, 0.07470867037773132, 0.4118864834308624, 0.3178292512893677, 0.001793531933799386, 0.12013185769319534, 0.7819526791572571, -0.2888803780078888, -0.23373207449913025, 0.8873742818832397, -1.6043930053710938, 0.6584348678588867, 1.0120402574539185, 0.1642705649137497, -0.5039325952529907, 0.855444073677063, -0.9840871095657349, 0.3774344027042389, -0.4185231626033783, 0.993054211139679, 0.9034820795059204, 0.7932164669036865, -1.3702870607376099, -0.6906594634056091, -0.6241178512573242, 1.23329496383667, 0.1776667982339859, -0.1476389318704605, 2.049436569213867, -1.4925131797790527, 1.9260295629501343, 0.7147171497344971, 0.659079909324646, 0.8594619035720825, -1.1966552734375, -0.30557623505592346, -0.31601041555404663, 0.43367433547973633, -1.1845903396606445, 0.632522702217102, -0.08606647700071335, 1.0248167514801025, -0.07378307729959488, -0.7916604280471802, -0.42341846227645874, 1.2561205625534058, -0.5595505833625793, 0.21689274907112122, -1.0223770141601562, -0.4902547001838684, 0.9069807529449463, 1.644715428352356, -1.0372613668441772, -1.2681355476379395, 0.44975408911705017, 1.9268187284469604, -0.6033079624176025, 0.13495439291000366, -0.7611777186393738, 0.6618935465812683, -2.3358378410339355, -0.6427533030509949, 0.1193198412656784, -0.8914000988006592, -0.6492183208465576, -0.15668818354606628, -1.1951148509979248, -0.41004621982574463, 1.8183320760726929, -0.18347987532615662, -0.5475170612335205, 0.14020664989948273, 0.6619192957878113, -0.43821433186531067, -0.5907373428344727, 0.9319355487823486, 0.4073753356933594, 1.9167912006378174, 0.43074458837509155, 0.10857030004262924, -0.2763340175151825, -0.7146568894386292, 0.8359060883522034, 0.20916038751602173, 0.6733997464179993, 0.3978149890899658, -0.9092327356338501, -0.11355053633451462, -0.42714306712150574, 2.752481698989868, -1.2889386415481567, 0.7082101702690125, -0.2724059522151947, 1.0969363451004028, -0.6073299050331116, 1.3317087888717651, 0.24661670625209808, -0.8744215369224548, 0.10949603468179703, -0.9968148469924927, 0.2072499692440033, 0.5482878088951111, 1.1889601945877075, 0.9392228126525879, -0.878230631351471, 2.1805500984191895, 0.4496923089027405, -2.958486318588257, -1.4659852981567383, 0.8779475092887878, -0.6745961904525757, 0.26724162697792053, -1.0402601957321167, -0.6288191676139832, -0.2734937071800232, -0.2246927171945572, 0.5221667289733887, 0.4840841591358185, -0.13241703808307648, -0.10649847984313965, -0.6955437660217285, 1.207607626914978, 1.4192208051681519, 1.359378695487976, 0.8781969547271729, -0.387265682220459, -0.7911658883094788, 0.23433801531791687, -1.2779990434646606, 2.046358823776245, 0.1412908285856247, 0.1292741298675537, 0.2857781648635864, 0.8051970601081848, 0.0095148254185915, 0.254881888628006, -0.71760493516922, -0.906029224395752, 0.795071542263031, -0.6898430585861206, 0.01273046713322401, 0.7445731163024902, 0.8677483201026917, -0.7795284390449524, -0.8192716836929321, 0.34276941418647766, -0.4672088325023651, 0.6897658705711365, 0.14470519125461578, -0.21765021979808807, -0.40023523569107056, 0.8709365725517273, -0.013465573079884052, -0.5388695001602173, 0.4071989357471466, -0.022119970992207527, 0.035330165177583694, -0.6305696368217468, -0.16031897068023682, 1.695791244506836, -2.192291736602783, -0.20297719538211823, 0.23028923571109772, -1.3596841096878052, -0.36790499091148376, -0.13246740400791168, 1.7857627868652344, -0.6989418268203735, -0.18263153731822968, -0.1292443722486496, -1.58037269115448, -0.9929384589195251, -0.3433472216129303, 0.3064954876899719, 0.4905185103416443, 0.9369134902954102, -0.16623686254024506, -1.502875804901123, -1.3501315116882324, 1.5272713899612427, 0.4962373673915863, 1.668312430381775, -0.020236888900399208, 0.1398105025291443, -0.32300132513046265, 0.5763674378395081, 0.6943219304084778, 0.23596782982349396, 1.3600635528564453, 0.42587795853614807, 2.5586023330688477, 0.7592663764953613, 0.07661137729883194, 0.47894421219825745, -2.0339407920837402, -1.4076906442642212, -0.1577117145061493, 0.4433785080909729, 0.6808913350105286, -0.043792206794023514, -1.1664241552352905, 1.0321426391601562, 0.7951071262359619, 0.4448114037513733, -1.3827223777770996, 0.548382580280304, 1.4905654191970825, -0.6970323920249939, -1.5167442560195923, 0.23056960105895996, 0.5119431018829346, 0.39821377396583557, 0.14215581119060516, -2.4537150859832764, 1.6870019435882568, -0.3735646903514862, -0.5784702301025391, 0.7612553238868713, -0.6777247786521912, 1.0469282865524292, 0.4824638068675995, -1.3416681289672852, -0.35318297147750854, -1.0162622928619385, -0.3288254737854004, -1.4537665843963623, 0.15904097259044647, 0.7239461541175842, -1.3447612524032593, 1.6654722690582275, 0.10682640969753265, -2.8457958698272705, -0.2952466607093811, -0.7101821899414062, -1.0971570014953613, 0.9579328894615173, 1.1521146297454834, 2.0597965717315674, 0.292988121509552, -0.2324620634317398, -0.09767203778028488, 1.1809226274490356, -1.0091882944107056, -0.44885921478271484, -0.509933590888977, 0.8205791115760803, 0.37060773372650146, -1.4327524900436401, -1.7773606777191162, -2.2161788940429688, 1.1407076120376587, -0.7992381453514099, -0.5690293312072754, 0.7908845543861389, 0.224832683801651, 0.15473569929599762, 1.5936087369918823, -1.0689834356307983, 0.9402174949645996, 1.0278240442276, -0.31707459688186646, -0.669977068901062, 1.2034307718276978, 0.19236122071743011, 0.47781088948249817, 0.18321475386619568, -0.4500507116317749, 0.8578735589981079, -1.0062085390090942, -0.9372273087501526, -0.23107299208641052, 0.40821877121925354, 0.5559526681900024, 0.7054659128189087, -0.46884045004844666, -1.1250132322311401, -0.0819561704993248, 0.3660680651664734, 0.466279536485672, 0.44992968440055847, 1.0567444562911987, 0.429109126329422, 0.3014739453792572, -1.3285444974899292, -1.0626704692840576, -0.08814442902803421, 0.4545386731624603, 0.33059409260749817, -0.590355396270752, -1.1363054513931274, -1.2592225074768066, -0.5102749466896057, 1.7974964380264282, 1.146412968635559, 0.44931548833847046, -0.17463262379169464, -1.26612389087677, -1.1724368333816528, 0.3458055853843689, 0.9534022212028503, 0.42875146865844727, 1.6044267416000366, 0.825469970703125, 0.49226051568984985, -0.6819437742233276, 1.9594793319702148, -0.06456045806407928, -1.0356903076171875, 0.7369919419288635, 0.5117969512939453, 0.2635168433189392, 1.1345667839050293, 1.4409109354019165, -1.6488475799560547, 0.3543674647808075, 0.9684189558029175, 0.2778015434741974, -0.6680410504341125, 0.037559717893600464, -0.58480304479599, -0.7209522724151611, -0.49068379402160645, -0.14519666135311127, 2.3707950115203857, -0.5147102475166321, 1.5793441534042358, 0.46002474427223206, -1.3985930681228638, 0.9118283987045288, -0.8951386213302612, 2.1773102283477783, 0.481459379196167, 0.18365658819675446, -0.6213614344596863, 2.171383857727051, -0.829678475856781, -1.2508479356765747, 0.21854496002197266, -0.9978298544883728, -1.3051637411117554, 1.3249313831329346, 0.7060776948928833, -1.5170788764953613, 0.8887243270874023, -1.176317572593689, -1.1699204444885254, -1.3477487564086914, -0.5866566896438599, -0.6864243745803833, 0.5671088695526123, 0.9859029054641724, 2.0542945861816406, -0.8577631711959839, 0.7216081619262695, 1.787842035293579, 0.6165273189544678, -0.7822587490081787, 0.009108920581638813, 1.1230484247207642, -0.5704524517059326, -0.7201139330863953, -0.7711082696914673, 0.3280794322490692, 1.4180527925491333, -0.3479032814502716, 0.7730473279953003, 0.13151226937770844, -1.1932742595672607, -0.8349499702453613, -0.8052438497543335, 1.0981500148773193, -0.6276445984840393, -1.0369843244552612, -0.676764190196991, 1.5088648796081543, 1.758847951889038, 1.337568998336792, 0.002868453972041607, -0.3871738612651825, 0.0800633504986763, 1.446818470954895, -0.30895400047302246, 1.013846516609192, -1.0464279651641846, 1.3420121669769287, -0.8279904723167419, -1.2268378734588623, 0.30343472957611084, -0.970112681388855, 1.3862881660461426, -0.43702012300491333, 0.400076299905777, -0.7841619849205017, 0.48162972927093506, 1.1364548206329346, 0.5191763043403625, -0.251020222902298, -0.13704347610473633, 0.19510067999362946, -0.674837052822113, -0.4822610020637512, -1.0266122817993164, 0.7828152775764465, -2.99788761138916, 0.44105830788612366, -1.2157979011535645, 0.8260632753372192, -0.5994316935539246, 0.19765567779541016, 2.130235195159912, 0.35039788484573364, 0.19340083003044128, -0.56296306848526, 0.5203696489334106, 1.0123953819274902, -1.339229702949524, 0.10622357577085495, -1.3570607900619507, 1.3643501996994019, -0.5306696891784668, -1.268815279006958, 0.5908495187759399, -0.8443303108215332, 1.1162770986557007, 0.6532588601112366, -0.31394636631011963, 0.2923993170261383, 0.027290387079119682, -1.3783891201019287, -2.004040241241455, 0.30668115615844727, 0.3996061682701111, -0.28274452686309814, -1.7596009969711304, -2.1456735134124756, -0.3787156045436859, 1.099666714668274, 0.36883389949798584, 0.16351252794265747, -0.6655088663101196, -0.5380022525787354, 0.9876841306686401, -1.2573412656784058, 1.119530439376831, 1.2794640064239502, 1.1551380157470703, 0.3026331961154938, 0.5176275372505188, -0.8127350807189941, 0.12000599503517151, -0.09347347915172577, -1.5842300653457642, -0.8009520173072815, 1.5946921110153198, -1.068373441696167, 0.02112429216504097, 0.32631123065948486, 0.5367956161499023, 0.7481307983398438, -0.01569492369890213, -1.9485561847686768, 1.0978717803955078, -1.4126168489456177, 1.4830191135406494, 1.5312039852142334, -1.1029133796691895, 1.1814640760421753, -0.4611515402793884, 0.6526166200637817, 0.8982605338096619, -0.4845793545246124, 0.4162653088569641, 0.110117606818676, -0.8133956789970398, -0.945990800857544, -1.2418370246887207, -0.4513154625892639, 2.1455161571502686, 0.326641321182251, 0.9042969346046448, -0.9798151254653931, -0.03585885092616081, -1.0086052417755127, 0.7772108316421509, -0.0018619445618242025, 0.5417255759239197, 0.3857896029949188, -0.7606015801429749, 1.0256140232086182, 2.6410765647888184, 1.4053438901901245, -0.4903118908405304, -1.6089764833450317, -1.383862853050232, 1.172160029411316, 1.4992247819900513, 1.1747967004776, -1.448710322380066, -1.2461951971054077, -1.064192771911621, 1.2054065465927124, -0.34582436084747314, -1.453984022140503, -0.6351256370544434, 0.8654395937919617, 0.22359848022460938, -1.5397812128067017, 0.19853559136390686, -0.8633589744567871, -0.3753710091114044, 1.2449829578399658, 1.4870612621307373, 0.4765932857990265, 1.559424877166748, -0.5700756907463074, 0.3794431984424591, 0.494005024433136, 0.6605632901191711, -0.4239669740200043, 0.49737218022346497, -0.6940436363220215, 0.35085105895996094, -0.4989546537399292, -1.8489656448364258, 0.878322184085846, 1.3977891206741333, 2.052330493927002, -1.7135084867477417, 0.2717752456665039, 0.8368203043937683, -0.7218480706214905, 0.6409329771995544, -0.7505320310592651, 0.5200831890106201, -0.24875320494174957, 1.1230286359786987, -0.08740010857582092, -0.35903552174568176, -2.550194263458252, 0.04963154345750809, -0.6216325163841248, -0.384065181016922, -1.6134239435195923, 0.16761046648025513, -0.745148241519928, -1.0084552764892578, -2.3559350967407227, 0.5737894177436829, 1.3410311937332153, 1.231166958808899, -0.18741178512573242, -0.5539113879203796, -1.6084929704666138, 2.6899869441986084, 1.179386854171753, 1.446900486946106, 0.19720768928527832, 0.006201219744980335, -0.8563925623893738, 2.5279011726379395, -0.9462449550628662, -0.036364153027534485, -0.37290918827056885, 0.886613130569458, 1.0656459331512451, 0.39609330892562866, 0.5158743858337402, -1.0471928119659424, 0.6569761633872986, 1.8275190591812134, 0.37977418303489685, 1.0820468664169312, 0.021397177129983902, 0.5336923599243164, -1.1427459716796875, -0.6010110378265381, -1.064652681350708, 0.8129242658615112, 1.3114964962005615, -1.2444027662277222, 0.3420347571372986, 0.43228647112846375, 0.5407335162162781, -1.3982926607131958, 0.5256337523460388, -0.38169553875923157, -1.0646058320999146, -0.5785853266716003, 0.45105257630348206, 0.560303270816803, -0.21236178278923035, -0.41207996010780334, -0.07385260611772537, -0.8303602337837219, 0.052392952144145966, 1.018323302268982, 0.7935689687728882, -1.733612060546875, 0.18011227250099182, 0.24224460124969482, 1.3566114902496338, -1.087782382965088, 1.3307982683181763, 0.7254830598831177, -1.381213665008545, 0.46205422282218933, 0.3965459167957306, 0.39175817370414734, -0.12134105712175369, -0.951201856136322, 0.2603348195552826, -0.012969466857612133, -0.8000106811523438, 0.10217496752738953, -1.036617636680603, 1.1831437349319458, 1.4945144653320312, 1.2619667053222656, 1.2615413665771484, 1.2376700639724731, 0.2786921262741089, 0.41915008425712585, 1.299329161643982, 1.2798823118209839, -1.4168633222579956, -0.4732692241668701, -0.6477673649787903, 0.6508032083511353, 0.4100275933742523, 1.4085439443588257, 0.22590371966362, 1.4008146524429321, -1.4862264394760132, -0.871440052986145, 0.46020084619522095, 1.3422743082046509, -0.7873256802558899, -1.3634954690933228, 1.8622263669967651, -1.248255968093872, -0.09613052010536194, 1.2137774229049683, 0.7647683024406433, 0.77164626121521, -1.1926852464675903, 0.9759151339530945, 1.0205128192901611, -0.4425218403339386, -0.4046858251094818, 0.09907571226358414, -0.44371864199638367, 0.9871507287025452, 0.019528629258275032, 1.231675386428833, 1.0977226495742798, 0.35596147179603577, 0.26273369789123535, 0.3200792074203491, -1.794479489326477, -0.04890461266040802, -0.12369920313358307, -0.5579989552497864, 0.018268313258886337, -0.734353244304657, 0.12004800140857697, 1.8454424142837524, 0.4983857572078705, -0.6548129320144653, 0.8992701768875122, 0.030824175104498863, -1.1867080926895142, 0.14410226047039032, 0.18042755126953125, 0.797435462474823, 0.5633954405784607, 0.5803760290145874, -0.15316718816757202, -0.10900094360113144, 0.7250897288322449, -0.7328969836235046, -2.812917709350586, 0.2281835824251175, 0.7276580333709717, 1.833146095275879, 0.7451893091201782, 0.04149538651108742, -0.11382796615362167, -0.5204634070396423, 2.7041678428649902, -0.19691288471221924, -1.097403883934021, 0.690238356590271, -0.9906353950500488, 2.496150016784668, 1.5668666362762451, 1.2754547595977783, -0.12541072070598602, 0.6796109676361084, -0.7236406803131104, -0.6528740525245667, 0.49430766701698303, -0.11349498480558395, -0.23957981169223785, -0.5169979929924011, 0.4926731288433075, -1.1552072763442993, -0.7885440587997437, 1.0873337984085083, 0.01824754662811756, 0.8597211837768555, -0.6354960203170776, -0.26894915103912354, -0.29496312141418457, 0.32315483689308167, 1.2542349100112915, -1.2871912717819214, -0.12697024643421173, 0.718113124370575, -0.4031720459461212, -2.1753249168395996, 1.1168371438980103, -0.13776423037052155, 0.4195382595062256, 0.9836605787277222, 2.2621536254882812, 0.8533294200897217, -0.8436818718910217, -0.6516042947769165, -2.152395725250244, -0.6543169021606445, -1.4976376295089722, -0.42014679312705994, -0.25076550245285034, 0.32967519760131836, 1.7619976997375488, -0.6192929744720459, 0.557062566280365, 0.964338481426239, 1.2400755882263184, 1.0210914611816406, -0.7723974585533142, 0.5644581317901611, -0.4124161899089813, 1.1088802814483643, 1.0238052606582642, 0.5409448742866516, -1.1127023696899414, 2.191190719604492, 1.3019837141036987, 0.557627260684967, 0.7618139386177063, -0.8875402808189392, -0.17723682522773743, -1.7636396884918213, -0.18690988421440125, 2.4874091148376465, -1.1342759132385254, 0.7244150042533875, 0.2166569083929062, -0.12369734048843384, 2.213304281234741, 0.3145008981227875, 1.588984727859497, -2.1415469646453857, 0.3218609094619751, -0.693598747253418, -0.17012982070446014, 0.8049169778823853, -1.5095230340957642, -0.5945684909820557, 1.196266531944275, 0.24246816337108612, -1.3013840913772583, 1.0444459915161133, -0.41658321022987366, -0.010796843096613884, -2.6322999000549316, 0.9524590373039246, -0.37338417768478394, -0.8302562832832336, -0.8402378559112549, 2.206173896789551, -1.2766716480255127, 0.8330190777778625, -1.4752986431121826, -0.24731220304965973, 0.9405625462532043, -0.21596257388591766, 0.0013753228122368455, -0.7434746623039246, 0.690204918384552, -2.2573482990264893, 0.29038047790527344, 0.9743432402610779, 1.8526346683502197, 0.972466766834259, 0.35075655579566956, 0.8408929109573364, 0.8101409673690796, -1.4969998598098755, -1.315348744392395, 0.7406995892524719, 0.3839667737483978, 0.9751590490341187, -0.9649304151535034, -1.393855333328247, -0.8001126646995544, 0.659567654132843, 0.7206766605377197, -0.739713191986084, -0.9867494106292725, -1.1230478286743164, -0.8151408433914185, -2.0725455284118652, -1.2950599193572998, -2.1754534244537354, -0.7549185156822205, -0.9830102920532227, 0.2472190111875534, -0.5828864574432373, 0.15115097165107727, 0.5077596306800842, -0.3599871098995209, -0.5787209868431091, 0.4371897876262665, -0.7695454955101013, -0.9335417747497559, -0.782724916934967, -0.2125953882932663, -0.7235625982284546, 0.9394636154174805, 0.6590319871902466, -2.6467208862304688, -1.9017695188522339, -0.6212123036384583, 0.27350983023643494, -1.6572284698486328, 0.42992618680000305, 1.168946623802185, -0.03839545324444771, 0.8140275478363037, 0.31965547800064087, -0.3888917863368988, 0.5174340009689331, -0.07510097324848175, 0.28872254490852356, -2.315868377685547, 0.4527876675128937, 0.5878127813339233, -0.1539134532213211, 0.5113121867179871, 0.10978124290704727, -1.4314777851104736, 0.6406102776527405, 0.2850435972213745, 0.058577362447977066, 1.6319348812103271, 0.7590681314468384, 0.6537747979164124, -1.2309751510620117, 0.14306005835533142, 0.19143320620059967, 0.439836710691452, 0.1136946901679039, 1.1987357139587402, 1.0522464513778687, 0.9199974536895752, -0.0230009276419878, 1.4505374431610107, -0.500391960144043, 1.0773674249649048, 0.18674471974372864, -1.5826213359832764, -0.7410200238227844, 0.4041990637779236, 0.8078101277351379, 0.3476668894290924, -1.2205575704574585, 0.20519942045211792, -0.16934506595134735, 0.14648650586605072, -1.4533982276916504, 0.1457938402891159, 0.4808617830276489, 0.32311543822288513, 0.6634132266044617], [-0.19850173592567444, -0.17393258213996887, -0.28815901279449463, 1.1313196420669556, 1.7510310411453247, 0.7340811491012573, -0.4345989227294922, 1.4203102588653564, 0.35030636191368103, -1.5797741413116455, 2.5898218154907227, -1.3595610857009888, 1.0118407011032104, -2.2008023262023926, 0.39347824454307556, -1.0461266040802002, 0.27529364824295044, 1.3632925748825073, -0.2999466061592102, -2.4036498069763184, 0.2026231288909912, 1.2846754789352417, -0.6096020936965942, 1.0712168216705322, 0.10395727306604385, 1.1769230365753174, -0.7338440418243408, -0.3144749701023102, 1.757948637008667, -0.06694403290748596, 0.1630989909172058, 0.4614729881286621, -0.7073307633399963, -1.1745903491973877, 0.030407773330807686, 0.30123037099838257, -1.5433461666107178, -2.389158248901367, -0.3345303237438202, 1.0169575214385986, 0.8788337111473083, 0.2570536732673645, -0.7253440022468567, -0.8741796016693115, 0.22011201083660126, 0.3548871576786041, -0.09076149761676788, -0.05824492499232292, -2.8384196758270264, -0.6440137624740601, 0.9948217868804932, -1.8787367343902588, 0.6428384780883789, -0.40158164501190186, -1.2288421392440796, -0.6313095688819885, 0.6106866002082825, 0.19119791686534882, 0.7138436436653137, 1.7478107213974, -0.6679499745368958, -0.42195817828178406, -0.730113685131073, -0.25632089376449585, -0.8691928386688232, 0.7632209658622742, -0.38798677921295166, 0.6758151054382324, 0.1588650792837143, -0.5321359038352966, -0.08387014269828796, -0.00474746385589242, 0.8040434718132019, -1.1128114461898804, 0.7817725539207458, -0.5108241438865662, -0.7365027666091919, 0.13561378419399261, -0.38503116369247437, -1.2308820486068726, 1.4034662246704102, -0.5030775666236877, -1.0577523708343506, 0.8615967035293579, 0.9165902733802795, 0.8322556018829346, 0.7689622640609741, -0.8552971482276917, 0.4510127902030945, 2.7838871479034424, 0.9154332280158997, 1.6550573110580444, 0.4097871780395508, 0.714529812335968, 0.4489798843860626, 1.7602165937423706, -1.0947293043136597, -0.28611356019973755, -0.4357263743877411, -0.6999981999397278, -0.2677764296531677, 0.3757828176021576, -1.2434074878692627, -0.9779378771781921, 0.1970764547586441, -0.18831750750541687, 1.554329514503479, -0.433437705039978, 0.6758741736412048, -0.3610149323940277, -1.2107726335525513, 0.7397969961166382, 0.10867571085691452, 0.1251099705696106, 2.1638026237487793, 1.2674516439437866, 1.1887919902801514, -1.312411904335022, 1.4670239686965942, 0.47056159377098083, -1.4280427694320679, -0.899990439414978, 1.6797070503234863, 0.28090301156044006, 1.5271589756011963, -1.4406808614730835, 0.08149745315313339, 0.7627089023590088, -0.8398670554161072, 0.16397368907928467, 0.940024733543396, -0.4601363241672516, -0.4799836277961731, -0.597388744354248, 1.0027999877929688, -1.2503669261932373, -1.0743372440338135, -1.082082748413086, -1.0458788871765137, 1.7902511358261108, -1.168912649154663, 1.7771400213241577, -0.19122929871082306, 1.7137137651443481, 1.007861614227295, -1.5449447631835938, 0.881637454032898, -1.1185797452926636, 0.8592755794525146, 1.1311779022216797, 0.1880672127008438, 2.5797886848449707, -0.9208473563194275, -0.6451623439788818, 0.6273882389068604, -0.8527074456214905, 0.7371847033500671, 0.7943983674049377, 0.5942015647888184, -0.3608876168727875, -0.5067481398582458, -0.5474460124969482, 0.3557184636592865, 2.00579571723938, 0.04686800017952919, -0.43716365098953247, -0.4051833748817444, 0.013426163233816624, 0.8661965727806091, -0.24647504091262817, 0.4914032518863678, 0.09936828911304474, 0.6490499973297119, 0.8145272135734558, 0.04801752045750618, -1.6036572456359863, -0.5080169439315796, -0.549583911895752, 0.5797208547592163, 0.20691826939582825, 0.4838888943195343, 0.7835081219673157, 0.23266106843948364, -0.14483793079853058, -0.8536790609359741, 0.00928411353379488, 0.7508722543716431, -0.5768876671791077, -0.42358139157295227, -0.2953391671180725, -0.9767566919326782, 1.9207950830459595, 0.20605768263339996, 2.572849988937378, -0.19919203221797943, -0.5599302053451538, -0.35208651423454285, 0.33351272344589233, -2.213803768157959, 1.29572594165802, 0.32788634300231934, -0.18080437183380127, -0.24859607219696045, -1.1010491847991943, -1.1437232494354248, 0.7571613788604736, -0.9420611262321472, -1.0941874980926514, -1.0864078998565674, -0.37367358803749084, 0.6569859981536865, -0.12368156760931015, 0.5860003232955933, -0.8973642587661743, 0.47812047600746155, 0.6021603941917419, 0.8199911713600159, -0.3325527012348175, -2.518616199493408, -0.42118677496910095, 0.627558171749115, -2.4698126316070557, -1.4410017728805542, 0.8589120507240295, 1.274523377418518, -0.34586629271507263, 1.5683717727661133, 0.982580840587616, -0.49209603667259216, 0.42454954981803894, -0.32688724994659424, -0.6225597858428955, -0.76228266954422, 0.4556928873062134, -0.35412320494651794, -1.0581730604171753, -0.4361156225204468, 0.16957975924015045, 0.7659175395965576, 0.6969626545906067, -0.7638914585113525, -1.1406275033950806, -0.15538372099399567, -0.11524717509746552, 1.3171998262405396, 0.029154881834983826, -0.21365487575531006, 0.8336920738220215, 1.7309197187423706, 0.9059596061706543, -0.9443642497062683, -0.8745086789131165, 0.24081718921661377, 0.7557328343391418, -1.2071360349655151, -2.0942306518554688, 2.138478994369507, 0.014919840730726719, -0.10722190141677856, -1.2550575733184814, 0.8607518672943115, -0.5877780318260193, -0.20188014209270477, -0.5199029445648193, 0.398934930562973, 1.9420205354690552, -0.6005557775497437, -2.427386999130249, 0.42008501291275024, -0.5511826872825623, -0.23864960670471191, 0.4986179769039154, -1.0124120712280273, -0.5263519883155823, 1.3660168647766113, 0.4046436846256256, 0.1318882405757904, -3.713038444519043, 1.9067147970199585, -1.1235567331314087, 0.4855934679508209, 0.6663981080055237, 0.07758255302906036, 0.16035577654838562, 0.14269118010997772, -0.8695791959762573, -0.5096051692962646, -0.3454042077064514, -0.16683220863342285, -0.02196473442018032, 0.025722555816173553, 0.02129337750375271, -0.08599598705768585, 2.203413486480713, 1.0632312297821045, -0.6424505114555359, 0.6277634501457214, -0.06924988329410553, -0.24346311390399933, -1.2162035703659058, -0.5722282528877258, 0.624983012676239, 0.20293059945106506, -1.2856924533843994, -0.973771333694458, -0.4311727285385132, 2.015923500061035, 0.8018730878829956, -0.4405005872249603, -1.0418016910552979, 1.2575377225875854, 1.528746247291565, -0.6152562499046326, 0.8658722639083862, -0.47505590319633484, -0.16592568159103394, 0.13940131664276123, -2.4399280548095703, -0.5468164682388306, 0.6871641278266907, -0.5795105695724487, 1.168936014175415, -0.31340736150741577, -0.6362493634223938, 0.3045244812965393, -0.13391174376010895, 0.3222106397151947, 0.6555477976799011, 0.7763823866844177, 0.5638200640678406, -0.11123920977115631, -1.623191475868225, 0.2842637300491333, -0.5797024369239807, 0.11666668206453323, 0.6889281868934631, -0.9501972198486328, 1.3098152875900269, -1.3449851274490356, 0.9584741592407227, -0.8132496476173401, -0.038446903228759766, -0.04354147985577583, -0.16187535226345062, 0.07869423180818558, 0.12417428940534592, -1.9660109281539917, -2.0296471118927, 1.0106756687164307, -0.14773961901664734, 1.9161370992660522, 0.1499999463558197, -0.8269594311714172, -0.5984153747558594, 1.1761146783828735, 0.15120166540145874, -1.039506435394287, 0.6689972281455994, 0.2950681746006012, 0.17357708513736725, -1.6779088973999023, -0.5677592158317566, -1.4820948839187622, -0.7326173186302185, 0.21747076511383057, -0.8660908341407776, -0.6236715912818909, 1.0613774061203003, 0.9749023914337158, -1.2488727569580078, -1.1981725692749023, 0.15582957863807678, 0.976897120475769, 1.2631267309188843, 0.2433614879846573, 1.5434458255767822, -1.4516664743423462, 0.8412514328956604, 1.748281478881836, 1.0636178255081177, -0.9758780002593994, 1.537310242652893, -0.12955918908119202, -1.7853941917419434, 0.6755887866020203, 0.4987606406211853, -0.5027706623077393, 0.3105197548866272, 0.28111937642097473, 0.3343987464904785, 0.8201966285705566, 0.019162487238645554, 1.2015466690063477, -0.2959340214729309, 0.7951650619506836, -0.90130215883255, -2.6319148540496826, 1.152726173400879, 0.0006525090429931879, -0.5735669136047363, 0.33871665596961975, 0.40829694271087646, 0.2857663035392761, -1.0497857332229614, 1.3132052421569824, -0.7202768325805664, 0.6664155125617981, 1.4859634637832642, 0.012633376754820347, 0.07766798883676529, 0.6881221532821655, -0.10236683487892151, 0.06134547293186188, -0.9889604449272156, 1.3466541767120361, 1.0450105667114258, -1.5168287754058838, -0.17795398831367493, 0.21887482702732086, -0.8176001310348511, -1.4267419576644897, 0.7183510065078735, -0.11244788765907288, 1.0739256143569946, -1.418575644493103, -1.0440694093704224, -0.6826754212379456, -0.3323318362236023, 1.6064919233322144, -0.09270940721035004, -0.45519348978996277, 0.8287394046783447, -1.9833356142044067, -0.344126433134079, -0.7118015885353088, -0.6461931467056274, 0.7691866755485535, 1.029400110244751, 0.14941759407520294, -1.1202694177627563, 1.201683759689331, 1.116645097732544, 1.1116390228271484, -0.2785539925098419, 0.1940273940563202, 0.36496680974960327, -0.666526198387146, -0.21646934747695923, -2.996320962905884, -0.46113768219947815, 0.5429539084434509, 0.8954174518585205, -0.39417368173599243, 0.6502178907394409, 0.6092545390129089, -1.9667733907699585, -0.1711910218000412, -1.0121122598648071, 0.5685727000236511, -0.12274005264043808, 1.1368627548217773, 0.2292708307504654, 0.3047317564487457, -0.9105750322341919, 0.793948769569397, 0.3032130002975464, -0.13711245357990265, 1.9099079370498657, 0.8930760025978088, 0.5032627582550049, -0.4528612792491913, 1.1739392280578613, -1.094901442527771, -1.1330925226211548, -1.1388506889343262, -0.6176077723503113, -0.1565382331609726, -1.87239670753479, 0.15050549805164337, 1.1004172563552856, -0.7035248279571533, 0.6864995360374451, -0.21707749366760254, 0.6324337124824524, 2.3525078296661377, -0.30924084782600403, 0.7609016299247742, -0.7929804921150208, -0.0005203098990023136, 0.02171865478157997, -0.29899758100509644, -0.6865077018737793, -0.692976713180542, 0.06621280312538147, 1.0396066904067993, 0.4333905577659607, -0.4973105788230896, -1.0612505674362183, 1.6356643438339233, -0.3090248703956604, 1.3321548700332642, -1.0257492065429688, 0.10531451553106308, -0.9597334861755371, -0.43769171833992004, 2.120227575302124, -0.5232827663421631, 0.19361944496631622, 0.4287571310997009, 0.25565865635871887, 0.4082532525062561, 0.20911596715450287, 0.44859299063682556, 1.1782894134521484, 0.8077048659324646, 1.051798939704895, -0.8887647390365601, -0.9942885637283325, 1.1276003122329712, -0.9448426365852356, -0.9441767334938049, -2.8981833457946777, -0.917730450630188, 0.25304606556892395, -0.3704075217247009, 0.6642824411392212, 1.1444756984710693, -0.8800556063652039, 0.19066204130649567, 0.1703258603811264, 0.11505895107984543, 0.6492717862129211, -1.6386607885360718, -0.4475320279598236, 0.24692681431770325, -0.18630924820899963, -0.14323171973228455, 0.8510815501213074, 0.7582529187202454, -0.0012524682097136974, 0.20014415681362152, 0.09981304407119751, 1.2371944189071655, 0.1850355565547943, 1.079105257987976, -0.3280978202819824, -1.3094167709350586, 1.2659556865692139, 0.5870781540870667, -0.5760565400123596, 1.895159363746643, 1.2570035457611084, 0.3356424570083618, 1.067008137702942, 0.8236663341522217, 1.639765739440918, -0.6783009767532349, 1.0468114614486694, -0.6535831689834595, -0.26869162917137146, -1.0784460306167603, -1.240797996520996, -3.059807300567627, -0.9746703505516052, -0.12447372823953629, -0.3287287652492523, -0.9424533247947693, -1.492170810699463, 0.7987377643585205, 0.17983582615852356, -0.8471888899803162, -0.955254852771759, -1.0311294794082642, -0.04768927022814751, 0.5155404210090637, 0.5637872815132141, -0.9408746361732483, 0.0903298482298851, -0.5573784708976746, 0.9215254187583923, -1.0416369438171387, -0.13193351030349731, -0.2666458487510681, -1.2449742555618286, 0.35550981760025024, -0.251693457365036, 0.9013704657554626, 1.140779972076416, -0.9499050974845886, 0.5373660922050476, 0.5500058531761169, -1.64821457862854, 1.550668478012085, -1.8272205591201782, -0.5549368262290955, 0.4568589925765991, 1.280112624168396, -0.13871780037879944, -0.06934012472629547, -0.8112756013870239, -0.0022086789831519127, -2.16188383102417, -0.3853447139263153, 1.4606293439865112, -1.326060175895691, -1.7867159843444824, 0.5064143538475037, 2.422945976257324, -0.9269158840179443, 0.7165626287460327, -0.2770952880382538, 0.6271266341209412, -0.4189753830432892, 0.1709928959608078, -0.9327378869056702, 0.42867526412010193, 0.6522121429443359, 0.4055633842945099, -1.18115234375, -1.7437474727630615, -0.7119525074958801, 0.6543914079666138, -1.7782167196273804, -1.1394599676132202, -1.618742823600769, -0.22662422060966492, -1.4729787111282349, -0.9671633839607239, 0.22374321520328522, -1.44998300075531, -0.1555497795343399, -0.3159967064857483, -0.5347388386726379, -0.4904990494251251, 0.7992501258850098, -1.7481173276901245, -0.5653350353240967, -0.3242974281311035, -0.5917395353317261, -0.12385653704404831, -0.8349426984786987, 2.5554003715515137, 1.782050371170044, 1.0776034593582153, 0.8337727785110474, -0.4511633813381195, 3.1311862468719482, 1.1155643463134766, 1.5358566045761108, -0.09685217589139938, 1.2186765670776367, 0.22877368330955505, -0.13475188612937927, -0.3374391198158264, -0.9199057817459106, 0.1981128305196762, 1.3430107831954956, 1.0952355861663818, 1.083858609199524, -0.3624497056007385, -2.70561146736145, -0.1974988877773285, 1.1218507289886475, -0.6421695351600647, 0.40206438302993774, 0.48409250378608704, 0.5768400430679321, 2.2977678775787354, -0.8994144201278687, 0.23896275460720062, -1.3172016143798828, -0.4560650587081909, 0.890273928642273, -0.11135058104991913, 0.3072235584259033, 0.5095344185829163, -1.0556085109710693, -0.8636921048164368, -0.7508751749992371, 1.0683307647705078, -0.964382529258728, -0.26875266432762146, 0.913678765296936, 0.4395752251148224, -1.1562960147857666, -0.7578063011169434, -0.4065069258213043, -1.066071629524231, 0.3141849935054779, 0.16539917886257172, 0.8052366375923157, 0.46742966771125793, -0.4206482470035553, 0.39826929569244385, -1.5535556077957153, -0.04172395542263985, 1.052098035812378, -1.1903870105743408, -2.371771812438965, 1.5890541076660156, -0.8058464527130127, 1.1211594343185425, -0.6165347695350647, 0.499408096075058, 0.6050873398780823, 0.7816363573074341, -0.09756634384393692, -1.5102511644363403, -1.8724493980407715, 0.1077466607093811, -1.520691156387329, -0.05794152989983559, -1.8049836158752441, 0.16349883377552032, 2.3450253009796143, 1.1532292366027832, -0.29358869791030884, 0.698423445224762, -0.9137891530990601, 0.1517920047044754, -1.1712987422943115, -0.6282070279121399, -0.6085773706436157, -0.06079191341996193, 1.3541343212127686, -0.21928094327449799, -0.0038715216796845198, 1.5406311750411987, -0.6211434006690979, -2.192732572555542, 0.6453573107719421, -0.9512514472007751, -0.5463852286338806, -1.349635362625122, 1.362769603729248, 0.9362305998802185, 0.6706820130348206, -0.9927910566329956, 1.190112590789795, -0.9204592704772949, 0.5241624116897583, -1.2619558572769165, 0.26670581102371216, -0.25545305013656616, -0.04097384959459305, -0.6977209448814392, 0.08837879449129105, 0.1043727770447731, -0.2794204354286194, 1.1941745281219482, -0.1900625079870224, 0.4713839590549469, -0.025760581716895103, 0.8400685787200928, 2.0879528522491455, -0.07536274939775467, -0.8795544505119324, 1.2148720026016235, -1.1226667165756226, 1.9813367128372192, 1.2051512002944946, 0.2674849331378937, 1.0488288402557373, -0.05187974497675896, -0.8378903865814209, -0.49004605412483215, 0.6843942999839783, -0.094423808157444, 0.5899815559387207, -0.40763920545578003, -0.4138927757740021, 0.2720629870891571, -0.9665431380271912, -0.010994530282914639, -0.7961381673812866, 0.03239695727825165, 0.561998188495636, 0.13090823590755463, 0.5352907776832581, -0.3795747458934784, 0.8487198352813721, 0.976464569568634, -0.5348420739173889, -0.4471333622932434, 1.1016368865966797, -1.242951512336731, 2.299386501312256, -1.2191327810287476, -0.6397185921669006, 1.1675186157226562, 0.6988313794136047, 1.6888970136642456, 0.9519193172454834, -0.5363457202911377, 1.6279072761535645, -0.06799530237913132, 0.05132189393043518, 0.8774240016937256, -0.3759320080280304, 0.38502591848373413, -0.38909998536109924, 0.43274661898612976, 0.22929753363132477, 0.343561589717865, -0.07643847912549973, 0.5910179018974304, 0.46656450629234314, 0.29481980204582214, -1.998640775680542, 0.11439801752567291, 2.4763317108154297, -2.0267601013183594, 2.3108198642730713, 0.7205172181129456, 2.521145820617676, -0.18282577395439148, 1.5868488550186157, 0.8049582839012146, -0.24588680267333984, 0.4382811188697815, 1.1563527584075928, -0.7340236902236938, 1.6754897832870483, -0.11055951565504074, 1.3388742208480835, 0.6110715866088867, 0.48914825916290283, -0.015245853923261166, 2.158425807952881, -1.9442980289459229, 1.6411634683609009, -1.344430923461914, -0.4386259615421295, 0.0423237681388855, -0.34134554862976074, -1.2818764448165894, -1.5802052021026611, 1.0144696235656738, -0.732383668422699, -0.2997474670410156, 0.05174136906862259, 0.22186630964279175, 1.9310784339904785, 0.9462482333183289, 0.8301966190338135, 0.06318016350269318, -0.27245432138442993, 1.0733301639556885, 0.7015684843063354, -1.3857905864715576, 0.16764479875564575, 0.7542687058448792, 0.19212888181209564, -0.1875169724225998, 1.2880973815917969, 0.8890054821968079, 1.0610578060150146, 1.0135267972946167, 0.6186712980270386, 0.19579766690731049, 0.8622726202011108, -1.176666259765625, -1.527083396911621, 0.17836695909500122, 0.5497397780418396, 0.978003978729248, -0.13998794555664062, 1.6540451049804688, -0.03240613266825676, 0.7633888125419617, -0.6351785063743591, 1.2385993003845215, 0.004199351649731398, -1.3938947916030884, -0.6180816888809204, 0.4123678505420685, -0.5083436369895935, 2.0382192134857178, 0.41073334217071533, -0.8989801406860352, -0.26878368854522705, -0.9297544360160828, -0.7474790215492249, -0.33506423234939575, 0.5409479737281799, 0.81871098279953, -0.7506620287895203, -0.3343183696269989, -0.34212252497673035, -0.5793038010597229, -1.07075834274292, 1.4289982318878174, -0.7186686396598816, -1.6961095333099365, 0.553096354007721, 0.3782687783241272, -1.0371447801589966, 1.5656987428665161, -1.4076435565948486, -0.42033347487449646, 1.3742332458496094, 1.283700942993164, 1.4528017044067383, -0.2917152941226959, 0.42591559886932373, 0.5862146615982056, -0.26868122816085815, 0.03775341436266899, 0.6353651881217957, 0.6583271026611328, -0.13985197246074677, 1.6898623704910278, -0.08317676931619644, -0.9852741956710815, 2.156754732131958, -0.6232578158378601, -1.0683534145355225, -1.0484845638275146, 2.0211427211761475, 0.40294983983039856, 1.4646018743515015, -1.6084650754928589, 1.0199404954910278, -0.47560131549835205, -0.8993922472000122, -0.9635825753211975, -0.7709048986434937, -0.9478248953819275, -0.2817520797252655, 0.292429655790329, -0.21910452842712402, -0.08144547045230865, 0.04814925417304039, -2.280369281768799, 1.166167140007019, 1.451345443725586, -0.12039513885974884, -1.376321792602539, 0.7085274457931519, -1.060271978378296, -0.6034914255142212, -1.570408821105957, 0.8957149982452393, -0.564922571182251, 0.0938412994146347, -1.0236506462097168, 1.3819694519042969, -1.5223549604415894, 0.07283910363912582, -0.7175400257110596, 0.2374817579984665, -0.6974362134933472, 0.3316026031970978, 0.9597370028495789, -0.21187162399291992, -0.9405669569969177, -0.8154730796813965, 1.1109341382980347, -0.9891348481178284, -1.445143461227417, -0.2692538797855377, -0.6169747114181519, -0.07091319561004639, 0.47528842091560364, -2.7168822288513184, -1.4147775173187256, 0.39070791006088257, -0.2943003177642822, 1.6715108156204224, -0.3478001356124878, 0.06398562341928482, -0.6851336359977722, -0.7385761141777039, 0.5978310108184814, 1.3551727533340454, 0.8813077807426453, 0.024097656831145287, -0.6735904216766357, 0.3868994116783142, -0.19347649812698364, 2.015106201171875, -1.3778845071792603, -0.5282110571861267, -0.5146517753601074, -1.2354413270950317, -1.6622869968414307, 0.2888448238372803, -0.25790631771087646, -0.7297877073287964, -0.4059891700744629, 1.289352297782898, -0.7517874836921692, -0.7288135290145874, -1.7254974842071533, 0.08736476302146912, 0.046115368604660034, 0.1446627527475357, 1.7973111867904663, -1.5571764707565308, -1.5586121082305908, -0.5613229870796204, -1.0104045867919922, 0.47614917159080505, 0.7954780459403992, -1.8343859910964966, 0.2854439914226532, 0.2431982010602951, -0.08884773403406143, 0.7017884254455566, 1.8241710662841797, 0.4775911569595337, 0.72442626953125, -1.016357421875, 1.2997534275054932, 0.44203728437423706, -1.0476961135864258, 1.229712963104248, -0.30729058384895325, 1.8478504419326782, -1.1677204370498657, -0.4714527726173401, 1.4314054250717163, -1.3182439804077148, -0.2668144106864929], [0.9133267998695374, 0.53205406665802, 0.3369932770729065, -1.5286821126937866, -1.3909345865249634, -0.693281888961792, 0.289132297039032, -0.4061175286769867, -0.485056072473526, -0.5017899870872498, 1.2635587453842163, -0.17618823051452637, 1.5488734245300293, -0.3538442552089691, -0.2762005627155304, 0.1268278956413269, -0.2011362910270691, -1.5639817714691162, -0.02984662726521492, 1.2481638193130493, -0.30188992619514465, 0.28275829553604126, 0.12139174342155457, 0.4081328213214874, 0.1196761503815651, 0.17528504133224487, -0.34477749466896057, 1.4547967910766602, -0.5906497836112976, 0.5562305450439453, -1.066640019416809, -0.7983242273330688, -0.9980372786521912, -0.2932007312774658, -1.2894965410232544, 1.5074028968811035, 0.07475379854440689, 1.2869638204574585, 0.25023695826530457, 0.05541885644197464, -0.18303819000720978, 1.6035164594650269, -0.9182253479957581, -0.9403677582740784, -0.47779834270477295, 0.326503723859787, 1.1748162508010864, -0.19872336089611053, -1.0801981687545776, 0.5805059671401978, 0.15540677309036255, -0.7574750185012817, -0.4374307096004486, -0.36321958899497986, -0.9334620833396912, -1.2541908025741577, 1.8275597095489502, -0.03083626553416252, 0.8949692845344543, 1.6208425760269165, -0.6878204345703125, 1.2285676002502441, 0.7206399440765381, -0.5742413401603699, -0.562724232673645, 0.8398874998092651, -1.200587272644043, 2.3313448429107666, -1.0660732984542847, -0.07969442754983902, -0.6316619515419006, -1.065696120262146, -0.7652872204780579, 0.9742119312286377, 1.1671394109725952, -0.11676733940839767, 0.08207421004772186, -0.1587897390127182, 0.5327243804931641, 0.2963564991950989, -1.8486814498901367, 1.7776340246200562, 0.4283159673213959, -0.20411893725395203, -0.381728857755661, -1.3574053049087524, -0.6044550538063049, 1.1042753458023071, -2.2942492961883545, 1.1768958568572998, 1.0862051248550415, 0.826964259147644, 1.1983120441436768, 1.0826170444488525, -1.579719066619873, 0.03471290320158005, 1.7266128063201904, -1.12149178981781, 2.53342604637146, -1.8169933557510376, 0.10465975850820541, -0.681644082069397, 0.3957994282245636, -0.531690239906311, -0.8326848149299622, -0.07256094366312027, 0.49260884523391724, 1.2839833498001099, 0.43967577815055847, 1.5719436407089233, 1.1099333763122559, 0.13796105980873108, 0.39912503957748413, -1.5900284051895142, -0.42141789197921753, -0.607204794883728, 0.4801313877105713, 1.46059250831604, 0.7208108305931091, -1.3646750450134277, 1.0769941806793213, -0.9076700210571289, -1.1034904718399048, -1.7638943195343018, 0.2486581951379776, -0.17305970191955566, 0.31267622113227844, -1.3628071546554565, -0.9477578997612, 0.6960986256599426, -0.20292115211486816, -1.650706171989441, 0.8815634250640869, -1.4369440078735352, 1.012120008468628, 0.4205864369869232, -0.35301458835601807, -2.0779664516448975, 3.36104416847229, -0.5664366483688354, -1.9373177289962769, -0.8218491673469543, -1.1128181219100952, -0.837628185749054, -1.5359092950820923, 1.1959284543991089, -0.9520165920257568, -1.4308245182037354, -0.6080076694488525, -1.1014404296875, 0.6182581782341003, -0.6325356960296631, 0.820969820022583, 3.050638437271118, 1.1315429210662842, 0.9296334385871887, -0.23667386174201965, -0.3014288544654846, 0.34810754656791687, -0.4988371431827545, 0.19779923558235168, -0.09218646585941315, -0.41589120030403137, 1.5879462957382202, 0.34328967332839966, 1.2032133340835571, 0.41690632700920105, -1.6948107481002808, -0.4332994818687439, 0.9242861270904541, -0.8552063703536987, 0.2279828041791916, -0.5557925701141357, 0.7053327560424805, -1.138216495513916, 1.6271841526031494, 0.9374540448188782, 0.15249206125736237, 0.7508472204208374, -0.5439854860305786, -1.763962984085083, 0.06609109789133072, -2.0377724170684814, -0.1555628478527069, -0.3975442051887512, 0.15534363687038422, 0.5592069625854492, -0.2849362790584564, 1.056787133216858, -1.0201191902160645, -0.4979652464389801, -1.7340476512908936, 0.23506850004196167, -0.5513408184051514, 0.3019004166126251, 0.38444364070892334, 1.8816454410552979, -1.4370062351226807, 1.0551751852035522, -0.5880467891693115, -0.9058892130851746, -0.43798908591270447, 0.09711406379938126, -1.1195451021194458, -1.5738474130630493, -3.0385584831237793, -0.5357872843742371, -1.7605335712432861, -0.0004173249180894345, -1.3964613676071167, -1.1944693326950073, 0.14522527158260345, 0.03373335301876068, -0.9201349020004272, 0.5627008080482483, 0.027347438037395477, -0.11995792388916016, -1.2115740776062012, -0.9209727644920349, -0.501399040222168, 0.5081467628479004, 0.16537396609783173, 0.05138596147298813, -0.3442295789718628, -0.8300169110298157, -0.26783716678619385, 0.2654447555541992, -0.13019339740276337, 0.8925436735153198, 0.683483362197876, -2.1197946071624756, -0.23890195786952972, 0.4923942983150482, -0.27634096145629883, 0.22982677817344666, -0.09498617798089981, -1.0683023929595947, 0.3459818363189697, 0.05100732669234276, 0.004644220694899559, -0.3509557247161865, -1.5029011964797974, -0.3934192955493927, -0.45785287022590637, 2.7574517726898193, -1.102795958518982, -0.6200237274169922, -0.8809528946876526, 0.10120450705289841, 0.37837740778923035, 1.1143780946731567, -1.238410472869873, 1.2656131982803345, 2.233537435531616, 0.35743698477745056, 0.5445430874824524, 1.2326672077178955, -1.6392085552215576, -0.5500379204750061, -1.0907814502716064, 0.4790108799934387, 2.3549091815948486, 0.4519236087799072, -0.837812066078186, 1.267776370048523, -0.44162917137145996, -0.8765875101089478, -0.6340137720108032, 0.5020226240158081, 0.2025652676820755, -0.6592242121696472, 1.906601071357727, 0.8093745112419128, 1.567488431930542, 0.4722460210323334, 0.43365469574928284, -0.894650399684906, -0.7365456223487854, 0.9564604163169861, 0.8269926905632019, -1.1649466753005981, -1.248960018157959, -0.20355623960494995, 0.41828653216362, 0.830542802810669, 0.16789144277572632, -0.3013731837272644, -0.25625866651535034, 0.8136818408966064, 0.751617431640625, -0.8826974630355835, -0.48002687096595764, -0.25550785660743713, -0.47648751735687256, -1.4616985321044922, 0.13149480521678925, -0.36350932717323303, 0.8153428435325623, -0.24117764830589294, 1.1766012907028198, 1.1845389604568481, 1.3333122730255127, -2.3934836387634277, 0.6920077800750732, -3.708706855773926, 0.5397777557373047, -0.380695641040802, 1.6511892080307007, 1.2427034378051758, 1.5763760805130005, 0.6576358675956726, 0.01302136667072773, -0.5568733215332031, 0.848331093788147, 1.4374840259552002, 1.3066534996032715, -0.8518878817558289, -0.8477469682693481, 0.9905118346214294, -0.8283323645591736, -0.20488394796848297, 0.4814956784248352, -1.920047402381897, 0.7232406735420227, -2.376021385192871, -0.952100932598114, -2.1315906047821045, 0.967352569103241, -1.4367923736572266, 0.9322996139526367, -0.9772257208824158, -2.437310218811035, -0.6184554100036621, 0.5994127988815308, -0.45555394887924194, 1.28328537940979, -1.456657886505127, -0.6591589450836182, 1.0102683305740356, 0.17864356935024261, 0.12407731264829636, 0.1181536614894867, 0.22506213188171387, 0.4268462061882019, 1.8016679286956787, 0.3010678291320801, 0.9543912410736084, 0.1131226196885109, 0.5469641089439392, 0.9868228435516357, -0.04470764473080635, 0.6361144781112671, 0.041329722851514816, -0.40154606103897095, 1.0722469091415405, -1.3213186264038086, -0.5886515974998474, -0.8777917623519897, -0.4893992245197296, 1.6019105911254883, -1.1155980825424194, -1.0057328939437866, 1.059816598892212, 0.7556737065315247, -0.9628573656082153, -0.373113751411438, -0.6714404225349426, 0.21680550277233124, -1.5316194295883179, 0.3647109270095825, 0.6788579225540161, 1.1572514772415161, 0.5123113989830017, -0.5766190886497498, -0.11930754035711288, -0.7233742475509644, 1.6646784543991089, -2.090805768966675, 0.6802645921707153, 1.7618049383163452, 1.1467269659042358, 0.07428769022226334, -0.7075856924057007, 1.8897792100906372, -0.23444688320159912, 0.3962072432041168, 0.43391257524490356, -0.7008777260780334, -0.1407853066921234, -0.33389174938201904, -0.9713647365570068, -0.49419522285461426, -1.0754420757293701, -0.10805726796388626, 0.906756579875946, -1.0557035207748413, -0.0713144838809967, -0.8302097320556641, 0.14929136633872986, 1.5696356296539307, -2.2438125610351562, 0.00879532191902399, -0.057129766792058945, -1.387951374053955, 1.6299599409103394, 0.03381461650133133, -0.5346417427062988, -0.39335840940475464, 0.26625701785087585, 0.9911219477653503, -0.17916281521320343, -0.25962379574775696, -0.6567015051841736, 0.10263767838478088, -1.965912103652954, -0.4053584933280945, -1.0602418184280396, -0.09219128638505936, 1.774249792098999, 1.549373745918274, -0.4933370351791382, -1.0565791130065918, 1.550594449043274, -1.1698687076568604, 1.2955549955368042, -0.6418473124504089, 0.7982581853866577, -0.4441741406917572, -2.0639467239379883, 0.7518260478973389, -0.2995164096355438, -0.8044757843017578, -1.6839152574539185, -0.12229536473751068, 0.28664258122444153, 0.44678592681884766, -2.0002238750457764, 0.12316595762968063, 0.6194970011711121, 0.649770975112915, 0.46090731024742126, -0.10058346390724182, -0.294298380613327, 0.3412671983242035, -0.7655215859413147, 0.7319726347923279, -0.17044515907764435, 0.6370921730995178, -1.517149806022644, 1.6509443521499634, -1.2384434938430786, 1.1564170122146606, 0.6736397743225098, -0.5488128662109375, 0.6970961093902588, 0.206644669175148, 0.7100556492805481, -0.5832815170288086, -0.25881028175354004, 0.1377369612455368, 0.4296607971191406, 1.0918737649917603, 1.1444257497787476, -0.08006004989147186, -0.9615426659584045, -0.30376097559928894, 1.3192378282546997, -1.2193272113800049, 0.5435073971748352, -2.190228223800659, 0.6511601209640503, -0.7214988470077515, -1.5948100090026855, -0.02351081185042858, -0.03221141919493675, 0.9610483646392822, -0.5736647248268127, -1.5501822233200073, -0.21311286091804504, -0.30965355038642883, -0.023145204409956932, 0.9508036971092224, 1.1838468313217163, -0.6694861650466919, -0.7454187870025635, 1.1292963027954102, -0.5660179853439331, -0.29793089628219604, -0.48987245559692383, 0.2606430947780609, 0.25768420100212097, -0.25602060556411743, 0.5214025974273682, -0.6045061945915222, 1.8158488273620605, 0.8687936067581177, -0.40597522258758545, -0.2407791018486023, -0.20893941819667816, -1.1622662544250488, 0.7133592367172241, 0.8542435765266418, -0.15440605580806732, 0.5409473776817322, 0.8483801484107971, -0.1931837648153305, 0.3543229401111603, 0.1529477834701538, -0.8911858797073364, 0.5883182883262634, 1.1006450653076172, -0.9927030205726624, -1.1106723546981812, 0.3805617094039917, -0.5046030879020691, 0.7201687097549438, 0.04602178558707237, 0.19053423404693604, -0.5763855576515198, 0.6444655060768127, -0.16776145994663239, 0.2062920778989792, -1.605861783027649, -1.2357823848724365, 0.6873995661735535, 1.7156057357788086, 1.3271628618240356, 1.0750601291656494, -0.3611557185649872, -1.1864145994186401, -1.9222145080566406, -0.5294969081878662, -0.5834053754806519, -0.16240346431732178, 1.5098037719726562, -1.2952615022659302, 1.3961830139160156, 0.6703064441680908, -1.2742871046066284, 1.5929861068725586, -0.13276199996471405, -1.925053358078003, 0.6092808842658997, -0.4614751935005188, -0.21255500614643097, -0.4850482642650604, -0.9556052088737488, 0.9036772847175598, 0.38611701130867004, 2.4950919151306152, 1.1998001337051392, -1.0350607633590698, -1.33099365234375, -0.062216244637966156, 0.1331620067358017, 0.3365313410758972, -1.5808594226837158, 0.8080458045005798, -0.03108527511358261, 2.1750636100769043, -1.464568853378296, -1.1757723093032837, -1.4173665046691895, 2.5208425521850586, 0.25741419196128845, 0.8875989317893982, -0.9707486033439636, -1.3374577760696411, -2.3238794803619385, -0.20283199846744537, -1.6566065549850464, 0.3991040885448456, 1.0736744403839111, 0.5579760074615479, -0.6579673290252686, -0.11844039708375931, -1.5327749252319336, 1.3941107988357544, -0.4730035066604614, 1.1535371541976929, -1.0189603567123413, 0.7983070015907288, -0.3109737038612366, -0.3038923144340515, -0.8149867057800293, 0.27253684401512146, 0.40102577209472656, 0.8130688071250916, 1.1847236156463623, 0.7806034684181213, -0.09300051629543304, 0.19066189229488373, -0.8705400228500366, 1.5729632377624512, 0.24272508919239044, 1.2322535514831543, 0.9906138777732849, -0.6977686285972595, 0.9602159261703491, 0.7051743865013123, -0.19755320250988007, -0.16499760746955872, 1.218266248703003, -0.09001404047012329, 0.5037333965301514, -0.20031695067882538, 0.14373652637004852, -0.05391548201441765, -0.5282406806945801, -0.5982484817504883, -0.08762027323246002, -1.6600242853164673, 0.08025016635656357, 1.6348587274551392, -1.0633841753005981, 1.0544893741607666, -0.9619215726852417, 0.08936097472906113, 0.7181900143623352, -0.11760468780994415, 1.5404142141342163, 0.372847318649292, -0.6786288619041443, -0.6058946251869202, -0.5823758840560913, -0.3084057569503784, -0.2751176357269287, 1.8654965162277222, 0.05635867267847061, 2.3632400035858154, 0.30034831166267395, 0.12013062834739685, 0.04338115081191063, 1.4371806383132935, 0.9905884861946106, -0.004272344056516886, 1.1886411905288696, -0.6944391131401062, -0.4617244601249695, -1.7479405403137207, -1.0672494173049927, -2.122343063354492, -1.2766163349151611, -2.001945734024048, -0.42645302414894104, -0.010989953763782978, -0.15518532693386078, -1.6387799978256226, 0.5656289458274841, 0.9200054407119751, 0.4607338607311249, 0.2668682932853699, 0.8536310195922852, 2.0412063598632812, -0.13227009773254395, -0.04939987137913704, 0.6145403385162354, 1.2934772968292236, 0.7171552777290344, 0.06933870166540146, 0.7756652235984802, -2.048672914505005, 0.8314637541770935, 0.1266867220401764, -0.2724056839942932, 0.6966187357902527, 0.9722262024879456, -0.6155308485031128, -0.18190087378025055, 0.6278939247131348, -0.031141413375735283, -0.7645914554595947, 1.6062511205673218, -0.5843544006347656, 0.008421139791607857, 1.1635403633117676, -0.8599369525909424, 1.3886955976486206, -1.4428026676177979, 1.0029447078704834, 0.3299159109592438, -0.45958778262138367, -1.6452972888946533, 2.580174446105957, 0.4003356695175171, -0.4573655128479004, -1.1734261512756348, 0.9304157495498657, 1.1731007099151611, -1.5880718231201172, -1.3430814743041992, -0.8852211833000183, 1.0277650356292725, -0.44101211428642273, -2.602987289428711, 0.0106082484126091, -0.9177828431129456, -0.10153509676456451, -0.08733978122472763, -1.5255802869796753, -1.6268839836120605, 0.7913352251052856, -0.011287332512438297, 1.829898715019226, 0.5214409828186035, 0.8295623064041138, -0.31393298506736755, 0.7961515784263611, 2.196455478668213, 1.4140369892120361, 1.0192501544952393, -1.1129647493362427, 1.4929819107055664, 1.7937065362930298, 0.6075103282928467, 0.970051109790802, 0.48449304699897766, -0.4566992521286011, -0.4877454936504364, -0.10724956542253494, 0.41010385751724243, -0.1515030562877655, 0.2697390019893646, 1.250465989112854, 0.6032587289810181, 0.14644983410835266, -0.975784420967102, -0.7351861000061035, 3.083747148513794, -0.22170458734035492, 0.3591369390487671, -1.5872831344604492, -0.010043716058135033, 1.0481611490249634, 0.33668506145477295, -1.097387433052063, 1.2634016275405884, -1.5057566165924072, 0.007212486583739519, 0.517754077911377, 0.6850726008415222, 0.7700696587562561, 0.5992894172668457, -0.6398968696594238, 0.1447218656539917, 1.1517046689987183, 0.8321878910064697, 1.0279664993286133, 0.14892423152923584, 0.2937298119068146, -0.38830798864364624, 0.8455794453620911, 1.4664249420166016, -1.3722996711730957, 0.5098153352737427, 1.0409036874771118, -1.7687296867370605, 0.7615439295768738, 0.7279173135757446, 0.1690228283405304, -3.5490846633911133, -1.0483477115631104, -0.8468540906906128, 0.43199416995048523, -0.4403846263885498, -0.4278959035873413, -0.4440458118915558, 1.566434383392334, 1.744791865348816, -0.4731306731700897, 0.7864007949829102, -0.48350584506988525, 1.4873746633529663, -0.25147461891174316, 1.3253042697906494, -0.7004345059394836, 0.44666799902915955, 0.6731459498405457, -0.5283077359199524, 0.3875989615917206, -1.649066686630249, -0.6022833585739136, -0.8461682200431824, -0.974843442440033, 0.02607666328549385, 1.8419092893600464, -0.819023072719574, -1.081484079360962, 1.0153398513793945, 1.424376130104065, 1.1681264638900757, 0.6872514486312866, -0.7186787724494934, 0.3041049838066101, -1.0344793796539307, 1.855664849281311, -1.0028982162475586, 0.5833743214607239, 0.47642481327056885, -0.45391684770584106, -2.1552746295928955, -0.28212884068489075, -0.4709669053554535, 0.24099226295948029, 0.10849711298942566, 0.16476064920425415, -0.22891822457313538, -0.47913745045661926, -0.9528347253799438, 1.1300005912780762, 1.4862209558486938, -0.7823578715324402, -1.9399586915969849, -0.00830206461250782, 1.0127372741699219, 0.18141065537929535, 0.35711193084716797, -0.3763647973537445, 1.639566421508789, -1.1297732591629028, -1.1461058855056763, -1.3424204587936401, -1.2537020444869995, 0.6858189702033997, -0.9607639312744141, -0.12686897814273834, -0.9381176829338074, 0.1308966428041458, 0.0227183997631073, 1.9761770963668823, -1.049783706665039, -1.2559313774108887, -0.7558592557907104, -2.170064687728882, -0.18691875040531158, -1.264879822731018, -1.003367304801941, 0.9691286087036133, 0.6014477610588074, -0.5006038546562195, 1.198351502418518, 0.38537681102752686, 0.42943501472473145, -0.558430016040802, 1.4749515056610107, 0.11809197813272476, 1.1535637378692627, -0.17151127755641937, -1.0091522932052612, -0.5199459791183472, -2.228415012359619, 2.563781499862671, 1.2257747650146484, -1.5765650272369385, 0.9157782196998596, 0.638359785079956, 0.18827582895755768, -1.3003040552139282, -0.8189172744750977, 1.626847267150879, -1.4229538440704346, -0.243655726313591, 0.07568433880805969, -0.8911782503128052, 0.6555305123329163, -0.80075603723526, 1.423337459564209, -0.9560564160346985, 0.539628803730011, -1.2324093580245972, -0.6445561647415161, 1.5519709587097168, -0.6837261319160461, -0.3827984929084778, 1.1105320453643799, -0.7490658760070801, -1.6308997869491577, 0.37626582384109497, 0.8895865082740784, -0.6714048981666565, 0.3669971227645874, 1.2539297342300415, 0.8066713809967041, 0.10465104132890701, -0.678705096244812, -0.18159961700439453, -1.2960056066513062, 1.7143323421478271, 1.2663708925247192, -0.8587822318077087, 0.7786989212036133, 0.09313398599624634, -1.7326395511627197, -0.9145590662956238, 1.4525251388549805, 1.216981053352356, 1.2401536703109741, -0.8578265309333801, -1.5866039991378784, 1.9457560777664185, 0.7429453730583191, 1.8664443492889404, -1.711008906364441, -0.916814923286438, -0.1269145905971527, -0.19310611486434937, -1.2189217805862427, 0.6646813750267029, 0.16861063241958618, -0.48194772005081177, -0.14747308194637299, 0.6493277549743652, -1.045885682106018, 0.38998323678970337, 0.22212626039981842, -0.5568013191223145, -1.2286946773529053, 2.683807134628296, -0.2981600761413574, 0.8046091198921204, -1.229893445968628, 0.6864396929740906, -1.4275175333023071, 0.18613123893737793, 0.3366272747516632, 0.6149194836616516, 0.6881676316261292, 0.2791832387447357, -0.7188604474067688, 1.1646409034729004, -1.3103684186935425, 0.6458595395088196, -0.7503098845481873, -0.3400856554508209, -1.2310761213302612, 0.5321266651153564, -0.07376031577587128, -0.06328167021274567, -0.5236131548881531, 0.2565050423145294, 1.3874731063842773, -2.0349316596984863, -1.4155480861663818, -0.6970692276954651, 0.5422943234443665, 0.1139296293258667, 0.6718897819519043, 1.3408044576644897, 1.6036102771759033, -0.05684527009725571, 0.07505206018686295, 0.445574015378952, 1.2195171117782593, -0.08858396857976913, -1.052260160446167, -0.9456342458724976, 0.9239954948425293, 0.012198875658214092, 0.8529859781265259, -0.5045475363731384, 1.1678928136825562, -1.6494687795639038, 1.3258740901947021, 0.3672436773777008, -1.6709504127502441, 0.776485800743103, -1.0818986892700195, 1.2774518728256226, -1.1918470859527588, -0.8062856197357178, 0.6149503588676453, 0.12506923079490662, -1.0163670778274536, -0.6859009265899658, -0.7100892663002014, -0.06304944306612015, -1.0067754983901978, 0.5530714988708496, -0.6319922208786011, -0.5188942551612854, 0.6152751445770264, -1.3730305433273315, 1.5406782627105713, 0.2964320182800293, -0.7629820108413696, 1.6514431238174438, 1.7538634538650513, -0.8234094381332397, 1.1342636346817017, -1.3266880512237549, 1.500885248184204, 0.6946607232093811, -0.7510795593261719, 0.6764463782310486, 0.5474939942359924, -0.726913332939148, 0.009617251344025135, -0.1450166404247284, 1.0481293201446533, -0.327671080827713, -0.759143590927124, -0.7940269112586975, -0.45240461826324463, 0.4134126603603363, 1.0405735969543457, 0.07275637984275818, 1.2071582078933716, 0.25579217076301575, -0.021762695163488388, 1.9110995531082153, 0.02872971072793007, -0.06629308313131332, -0.7533525824546814, -1.0692638158798218, 1.1349725723266602, -0.061835650354623795, -0.7193232178688049, -0.5437930226325989, 0.8771989345550537, -0.4001334607601166, -0.2861480116844177, 0.35492780804634094, -1.2860889434814453, 1.1415678262710571], [-1.9388116598129272, 0.38969656825065613, -0.2437494993209839, 0.0003160398337058723, 0.3375758230686188, -1.1399054527282715, -1.7150815725326538, -0.41354668140411377, 0.5207362174987793, -0.3440220057964325, 0.9560824036598206, 0.5312030911445618, -1.0335084199905396, 0.7742998600006104, -0.2157250940799713, 0.7051778435707092, -2.6938233375549316, -0.10358750820159912, -2.662992477416992, 0.48540976643562317, -0.18228602409362793, 1.879738688468933, 1.1944117546081543, 2.1909594535827637, 0.9069989919662476, 1.639534831047058, 1.8051340579986572, -0.7833449244499207, -1.0232661962509155, -0.5894660353660583, 0.7381475567817688, -1.5187679529190063, 0.38209331035614014, 1.1542606353759766, -1.048282265663147, -0.18003399670124054, -0.3865911364555359, -3.2752747535705566, -0.2566940188407898, 0.508171021938324, 1.7999016046524048, -1.693281888961792, -1.2244211435317993, -0.6850637197494507, 1.028407096862793, 0.055327367037534714, 1.923478603363037, 0.06839735060930252, 1.2676042318344116, 0.21216878294944763, -0.09683787822723389, 1.8799968957901, -2.30277943611145, -0.8655415177345276, 0.7374703884124756, 0.4000841975212097, -1.677560567855835, -0.9462481737136841, -0.5696262717247009, 0.36292025446891785, 1.2375086545944214, 1.319374680519104, 0.8427132964134216, -0.17546643316745758, 0.23270738124847412, -0.3699829578399658, -1.446547508239746, -0.7699483036994934, -0.1872095912694931, 0.11775075644254684, 0.8339589834213257, -0.6210430860519409, -0.1587805598974228, 0.6060241460800171, 1.0491949319839478, -0.4803183078765869, 2.720980405807495, 0.1317071169614792, 1.8048937320709229, -0.4669535756111145, 0.4879220724105835, -1.1244463920593262, 0.4339974522590637, 0.4756473898887634, -0.4092555642127991, 0.3789786100387573, -0.9540162682533264, 0.571461021900177, 0.8382044434547424, 0.018216511234641075, 0.38210663199424744, 0.30646973848342896, 0.3016013205051422, -0.18507379293441772, 1.8479455709457397, 0.2845686674118042, 0.06330776959657669, -0.2820814549922943, 1.8343284130096436, -0.6232002377510071, -0.21742933988571167, -0.43106284737586975, -0.41930922865867615, 0.47415968775749207, -1.2331526279449463, -1.591571569442749, -0.9974501729011536, 0.24983516335487366, 1.5614200830459595, 0.5619351267814636, 0.9184468984603882, -0.5554056763648987, -0.04746319353580475, 0.15530343353748322, 0.5321773886680603, -0.8120338916778564, -1.6505554914474487, -0.3403656482696533, 0.7771294713020325, -0.559228777885437, -0.07975064218044281, 0.7574564814567566, 0.07134509086608887, 0.5806329846382141, 0.08330255001783371, 0.35938671231269836, 0.37813258171081543, 1.0210729837417603, -0.4261336028575897, 0.9942789077758789, -1.2639265060424805, 1.298889398574829, -0.9016557335853577, 1.513015866279602, 0.36350759863853455, 0.9695410132408142, 0.8353654742240906, 0.8067786693572998, -0.8316742181777954, 0.1986418515443802, 0.25363126397132874, -0.10822214931249619, 0.2881043553352356, 3.0574300289154053, 0.7621226906776428, -1.1578160524368286, -0.2615783214569092, -0.8846749663352966, 0.2562829256057739, 1.628183126449585, -0.6075820326805115, 0.7029009461402893, 0.17976701259613037, -0.4817843437194824, -0.7695881128311157, 0.8860765695571899, 2.081465244293213, 1.9990553855895996, 1.5381922721862793, 1.5614511966705322, 0.43101051449775696, 0.4034876525402069, -0.8486358523368835, -2.6177361011505127, -0.45579174160957336, 1.2385694980621338, 2.014502763748169, -1.3601957559585571, 1.0069291591644287, 0.9228777289390564, 1.52291738986969, -0.13821710646152496, -0.7247036695480347, -0.07853920757770538, 0.5795979499816895, 0.6262249946594238, 1.6696264743804932, -1.9315447807312012, -0.4260419011116028, 0.7837023735046387, 0.28440791368484497, 1.256164312362671, 0.2679442763328552, -1.1065044403076172, -1.052063226699829, -0.8418827652931213, 1.8550032377243042, -0.033552415668964386, -1.487594723701477, -0.08459838479757309, 0.08208313584327698, 0.04890706017613411, 0.26689255237579346, 0.6978373527526855, -0.33956336975097656, -0.17871682345867157, -0.681594729423523, -0.48009634017944336, -1.6199941635131836, -0.5907444357872009, -0.32341495156288147, -0.49051666259765625, 1.5912001132965088, -0.5856810808181763, -1.9227558374404907, 0.28190693259239197, -1.0310505628585815, 0.11795122921466827, -2.271040678024292, -0.9121555089950562, 1.2084547281265259, -0.013225434347987175, 2.323577642440796, -0.2986610531806946, 1.253633737564087, 0.2105770707130432, 0.9313735365867615, -0.9909394383430481, 0.3494671583175659, -2.0050902366638184, 0.4974919855594635, 0.45671406388282776, -0.3211793303489685, -0.8769276142120361, 0.5113167762756348, 2.162444829940796, 1.4492460489273071, -0.6631737947463989, -0.05560244247317314, -1.9196650981903076, 0.596645712852478, -0.9661284685134888, 0.9712562561035156, -0.5123753547668457, 0.85481196641922, 0.27991336584091187, -0.8222455382347107, -0.10548543930053711, -1.8182727098464966, 0.07972099632024765, 0.9896884560585022, -1.4367249011993408, -0.7296378016471863, 0.2654356360435486, -1.1309642791748047, 0.690579891204834, 2.0725114345550537, 0.9248022437095642, -1.2023520469665527, 0.6571243405342102, 0.20284895598888397, -1.0738606452941895, -0.7734635472297668, -1.688435673713684, 0.1594184786081314, 1.4396510124206543, 0.7861053943634033, 0.5181825757026672, -0.7927648425102234, 1.1852627992630005, -0.44507625699043274, -0.5476042032241821, -0.8422315716743469, -0.40125545859336853, 1.047370433807373, 0.7464235424995422, 0.5129280686378479, -1.0602549314498901, -0.3588848114013672, -0.4908195436000824, 1.4895480871200562, 0.5906142592430115, -1.628492832183838, 0.38308364152908325, 0.6759129762649536, -1.6000081300735474, -1.0965640544891357, -1.4147069454193115, 1.1042759418487549, 0.04857328161597252, -0.7764624357223511, -0.078520767390728, -0.16061711311340332, 0.7064751982688904, -1.8620458841323853, 0.09573206305503845, -0.5397487878799438, 1.0324760675430298, -0.22371794283390045, 0.6467829942703247, -2.4093058109283447, 0.3507610261440277, -1.4507359266281128, -0.3356289863586426, -0.18433909118175507, 0.24599401652812958, -1.7167359590530396, -0.21783103048801422, -0.8056711554527283, -0.10904905945062637, -1.3527854681015015, -0.1256650686264038, -0.05391441285610199, 1.8274108171463013, -0.5381605625152588, 0.7285850644111633, -0.5619902014732361, -0.6219648122787476, -0.3989693522453308, -1.7699798345565796, -0.11853325366973877, -0.5391687154769897, -0.5706501603126526, -0.5621365308761597, -1.1431361436843872, 1.1840498447418213, -1.4178838729858398, 0.4377346932888031, 0.05859501287341118, 1.017809510231018, -1.447568416595459, 1.1981747150421143, 0.45913103222846985, -1.9554827213287354, 0.7527675628662109, -0.09910077601671219, -0.41868630051612854, 0.417182058095932, -0.8323625326156616, -1.4934930801391602, 0.3103283941745758, 0.8740545511245728, -0.3319748640060425, -0.23182660341262817, 1.078622579574585, 0.04865018278360367, 1.092342734336853, 0.7328423857688904, 0.14632093906402588, 2.5748062133789062, 0.6490709781646729, 0.41303905844688416, -0.15172433853149414, 0.412063330411911, 0.16906806826591492, 0.46847057342529297, 1.280559778213501, 0.7984448075294495, -0.13220299780368805, 1.36270272731781, -0.4192272126674652, 0.4035540223121643, -1.2623943090438843, -0.43230825662612915, -1.245348334312439, -0.04327413812279701, -1.10939621925354, 0.4324827194213867, 0.5532556176185608, 1.1399697065353394, -0.791731595993042, 0.4643422067165375, -0.5471122860908508, -0.8550857305526733, 0.3279305696487427, -0.9712619185447693, 0.1352824866771698, -0.7605818510055542, -0.9250771403312683, 0.3017280399799347, 0.8014194965362549, 0.08960035443305969, -0.11412979662418365, 0.7431386113166809, -0.006540755741298199, 0.4716218113899231, 1.6330558061599731, -2.3270232677459717, 1.5179451704025269, 0.08380413055419922, 0.10224168002605438, -0.35880839824676514, 0.2962571978569031, -0.25172340869903564, -0.35278940200805664, -0.09400641918182373, -0.8800678849220276, 0.37118518352508545, -0.6536788940429688, -1.7245888710021973, 0.6603205800056458, -0.15018384158611298, -0.07751224935054779, -1.7735379934310913, -0.8190121054649353, 0.2137192189693451, -0.14213915169239044, -1.5028021335601807, 0.2673427164554596, -1.560604214668274, 0.2868189811706543, 0.1040341779589653, -0.5311983227729797, -1.200221061706543, -0.12509191036224365, 0.22230295836925507, 0.5834421515464783, 0.8031203746795654, -0.7918259501457214, -0.17979750037193298, -0.6681334376335144, 0.4976898431777954, 0.2088787853717804, 0.19683249294757843, -0.5068061947822571, -1.3060009479522705, -0.3060627579689026, 0.7933399677276611, 0.8420685529708862, -1.5031510591506958, -0.2999180555343628, -0.8373251557350159, -0.3593505620956421, -0.46826064586639404, 0.28448551893234253, -0.3743106424808502, 1.5537984371185303, 1.5346378087997437, 0.4993407726287842, 0.9466159343719482, 0.510856568813324, 1.1116859912872314, -0.0006772938068024814, -0.19488757848739624, 0.6802740097045898, 1.1768604516983032, 0.8732128143310547, 0.6953139305114746, -0.5349918603897095, 0.8834530711174011, 1.0857460498809814, 0.04651353508234024, -1.221727967262268, -0.10222624987363815, 1.6436491012573242, -1.4404606819152832, 0.11487212777137756, -0.032763928174972534, 0.504938542842865, 0.5415111780166626, 0.5298328399658203, 0.1939280480146408, 0.1084408089518547, 0.08427030593156815, 1.3209726810455322, -1.2219732999801636, 0.08898179233074188, -0.18643496930599213, 0.9651615023612976, -0.5540104508399963, 0.5797843337059021, -0.20772038400173187, 0.2269163727760315, 2.295037031173706, -0.19814729690551758, 0.5737749338150024, 1.558035135269165, -0.8207657337188721, -1.2416131496429443, 0.5110000371932983, -1.7360870838165283, -0.26072630286216736, -0.37494710087776184, -0.36576399207115173, 1.2843314409255981, 0.2700810134410858, 1.0273306369781494, 0.5277408361434937, -0.18324413895606995, -0.1417125016450882, -0.5018159747123718, -0.9139266610145569, -1.9423397779464722, -0.5975407958030701, -1.6284655332565308, -1.9260790348052979, 0.6594476103782654, 1.960637092590332, -1.004494309425354, 1.2341183423995972, -0.8016602396965027, -0.57907634973526, 0.5664079785346985, -0.9462602734565735, 0.34373214840888977, 0.3328421711921692, -0.8174458742141724, -0.2556256353855133, -1.3355388641357422, -0.4931257665157318, 0.9721834063529968, 0.5936675667762756, 1.3873847723007202, 0.708107054233551, 1.1596366167068481, -0.3933770954608917, -0.2318429946899414, 1.2536338567733765, -0.4779195785522461, 1.9139727354049683, 0.6359463930130005, -0.8245877027511597, 0.3940753638744354, -0.45004624128341675, -0.45241689682006836, 0.7071864604949951, -0.265887975692749, -0.6689488887786865, -0.575045645236969, -0.32130298018455505, 1.3365392684936523, -0.9178059697151184, 0.45005500316619873, -0.8238842487335205, -1.1336504220962524, -0.7200684547424316, -0.8059226274490356, 0.3203313648700714, -0.5105761289596558, 0.24831701815128326, 1.8618673086166382, -1.405585527420044, -0.5454545617103577, -0.3054344356060028, 0.057024769484996796, -0.7779759168624878, -0.5740681886672974, -1.3299851417541504, 0.0520564541220665, 1.728977084159851, 1.0625594854354858, 0.4968406856060028, 0.08578888326883316, 2.2722830772399902, -0.6627712845802307, 0.19002120196819305, -0.2717515528202057, -1.2243916988372803, 2.3177433013916016, -0.7852077484130859, -0.08149929344654083, 0.7659479379653931, 0.7026445269584656, -1.2275277376174927, 0.8489242196083069, 0.42967742681503296, 0.28814247250556946, -0.4077465832233429, -0.03607628867030144, -0.47141867876052856, 1.2877689599990845, -0.11803461611270905, 1.089887022972107, 0.9261440634727478, -2.4093523025512695, 1.2564905881881714, -0.506187915802002, -1.5484387874603271, 1.683144450187683, 0.9401877522468567, -0.15160934627056122, 1.445409893989563, -0.4126291573047638, 0.14051179587841034, -0.3554626703262329, 0.36082473397254944, 0.24566659331321716, -1.0131795406341553, 1.876638650894165, -1.315130591392517, -0.6500353813171387, -0.5568275451660156, -1.1016356945037842, 0.8311527967453003, 1.1348509788513184, -0.9035981297492981, -0.873989999294281, -0.5425361394882202, 0.422453373670578, -1.162280797958374, -0.9544396996498108, 0.7071979641914368, -0.498088002204895, 0.36484286189079285, -0.962977409362793, -1.4245753288269043, 0.8982863426208496, -0.09458703547716141, -0.0009901814628392458, -1.8981553316116333, 0.6045951843261719, -1.2496742010116577, -0.5710306167602539, -2.8282241821289062, -0.9346761107444763, 0.4349461495876312, -0.06005965918302536, 0.9430261254310608, 0.35981884598731995, 1.1401809453964233, 0.05868859961628914, 0.43375203013420105, -0.10818949341773987, -1.0169992446899414, 1.9784852266311646, 0.2752135694026947, 1.0701091289520264, -1.1986637115478516, -0.649685800075531, 1.4032808542251587, 0.23025353252887726, -0.8217609524726868, 1.0396406650543213, 1.090161919593811, 1.4197407960891724, 0.6243433952331543, -2.270739793777466, -1.666898250579834, 1.673210620880127, -0.017387868836522102, -0.03490644320845604, 0.09407100081443787, 1.7125555276870728, -0.2778627276420593, -1.1901360750198364, 0.9695961475372314, -0.5242201685905457, 0.1497698277235031, 0.10512824356555939, -0.30774152278900146, 0.6163352131843567, -0.3183060884475708, 0.07354191690683365, -0.7286010384559631, 0.2184419333934784, -0.3206254243850708, -0.7323469519615173, 0.08632925152778625, 1.739416241645813, -0.5671983957290649, -2.1380953788757324, -0.4146164357662201, -1.0520328283309937, 0.3400927484035492, -1.5126527547836304, -0.35321244597435, -0.40012481808662415, 1.2343249320983887, 1.1428546905517578, 1.168520212173462, 0.9310488104820251, 0.9978960752487183, 0.05006727948784828, 0.29201990365982056, 0.16737599670886993, 0.30823707580566406, 0.9688399434089661, -1.0496947765350342, 0.3219122588634491, 0.3616303503513336, 0.06085215508937836, 0.829964280128479, 1.4969357252120972, -0.395829439163208, -1.1489779949188232, 0.4696402847766876, -0.2073078155517578, -0.395942360162735, -0.8803695440292358, 0.48273223638534546, -2.442201852798462, 1.4994547367095947, 1.6784335374832153, -0.03706051781773567, -0.9486243724822998, 0.5806072354316711, 0.12969297170639038, -1.011307954788208, -0.2831219434738159, -0.5001378059387207, 1.6380916833877563, 1.330501675605774, -1.158855676651001, -0.08398528397083282, -1.5762444734573364, 0.61744225025177, -0.22261738777160645, 0.004434278700500727, 0.11648459732532501, -0.4719078838825226, 0.4897480010986328, 0.7988198399543762, 0.6184014081954956, -0.7957654595375061, 1.7202568054199219, 0.3887383043766022, -0.9014725685119629, 0.02645791880786419, -0.31239598989486694, -1.3821969032287598, -0.6961153149604797, -1.3102858066558838, -0.16112448275089264, -0.2528006434440613, -0.6478254199028015, -0.23180732131004333, -0.5852527022361755, 0.4802263379096985, -1.1614197492599487, -0.5752738118171692, -0.5347622036933899, 1.5594066381454468, -0.6863049864768982, 0.05066614970564842, -0.3538070321083069, 1.1027714014053345, 0.7808830738067627, -0.3199971914291382, 0.2513793706893921, -0.7636466026306152, 1.4065808057785034, -0.6274275183677673, 0.3163590133190155, 1.5182909965515137, -1.2475218772888184, 0.5575666427612305, 1.3232800960540771, -0.4671531021595001, 0.29893913865089417, -0.409178227186203, 1.164799451828003, 1.5722302198410034, -0.7048740386962891, 0.6912580132484436, -1.4991931915283203, -0.8049647212028503, 0.4030507206916809, 0.7970375418663025, -0.12002657353878021, -1.3908886909484863, -0.05394040420651436, 0.03864959999918938, 1.0673987865447998, -1.6636801958084106, 0.01886684074997902, 0.5404257774353027, 0.3518563210964203, -0.32509908080101013, 0.8143410682678223, -1.5423508882522583, 1.0559934377670288, -2.2349202632904053, 0.17691192030906677, 0.36370593309402466, -0.2721271216869354, -1.5620787143707275, 0.22929011285305023, -1.0681803226470947, 0.9777862429618835, -1.5923460721969604, 0.8752740621566772, -0.8049927353858948, 0.7221924066543579, 1.4479477405548096, -1.9663969278335571, -0.5112150311470032, -0.8899123072624207, -2.409367084503174, 0.17974844574928284, 0.5321645736694336, 0.3032510578632355, -0.2699618339538574, -1.091755986213684, 0.005715834442526102, -0.8344357013702393, -0.25311043858528137, -0.292534202337265, -0.7350416779518127, -0.15038825571537018, -1.9529014825820923, -1.15849769115448, -0.5033977627754211, 0.1976991444826126, 0.5610867738723755, 0.6003304719924927, 0.7456762194633484, 1.4378961324691772, -0.950964629650116, 1.1196297407150269, -0.8870217204093933, -1.467246413230896, -0.3516360819339752, 0.8492202162742615, -0.595994770526886, 0.17415207624435425, 0.7076154947280884, 0.5503727793693542, -0.2221001237630844, 0.9165087938308716, 0.6064838767051697, 1.1094690561294556, -0.36770546436309814, -1.147487759590149, -0.5795390605926514, 1.5510666370391846, 0.6027884483337402, 1.0337897539138794, 0.4807322919368744, 1.3072466850280762, 1.4186543226242065, -0.5335788726806641, 0.2545454204082489, 0.3631923496723175, 0.5554442405700684, 0.6361207365989685, -0.589465856552124, -0.3980486989021301, -2.0637857913970947, -1.025926947593689, -0.6120588183403015, -1.3234171867370605, -0.9013656973838806, 1.7743722200393677, -0.859688937664032, 0.2675785422325134, 0.794969916343689, 1.0671648979187012, 2.272482395172119, -0.3274368643760681, -0.8857585191726685, -1.0044955015182495, -1.2012333869934082, 0.14965082705020905, -0.41178470849990845, -1.3847988843917847, -1.7082353830337524, 0.2843080759048462, -1.0184459686279297, 0.34633365273475647, 1.1723049879074097, -0.9607788324356079, -0.10320695489645004, 0.059520721435546875, -0.35487309098243713, -0.18042396008968353, -0.4849984347820282, 0.047963254153728485, -1.0184227228164673, -1.3877462148666382, -1.0787665843963623, 1.1918022632598877, -1.9518826007843018, -0.3903381824493408, -0.3446456789970398, -1.4379619359970093, 0.43669959902763367, -0.1377314180135727, -0.29385608434677124, 0.6215808987617493, -0.01974286139011383, 1.048677921295166, 0.018978139385581017, 0.5366858243942261, 0.4502728283405304, 0.596494197845459, 0.34794941544532776, -0.4392022490501404, 0.23533938825130463, -0.5257228016853333, 0.4645704925060272, -0.6428495049476624, -0.21633723378181458, 0.3761001527309418, -0.5843110084533691, -0.6386683583259583, 0.21226972341537476, -0.1725771576166153, 2.4134466648101807, -0.0667794793844223, -1.3634570837020874, 0.5460286736488342, -0.2346862554550171, -0.2506091594696045, 0.9300984144210815, 0.7552732825279236, -1.1690845489501953, 0.290808767080307, -1.7523192167282104, -1.0477346181869507, -0.6617369055747986, -0.7190505266189575, -0.42280474305152893, -0.32104161381721497, 0.590374231338501, -0.6799752116203308, -0.17226319015026093, -0.5763576030731201, -1.3722344636917114, 0.9490604400634766, -0.47169217467308044, -0.6621409058570862, -0.38822275400161743, -0.1719316989183426, 0.18533627688884735, 1.750705599784851, -1.0058830976486206, 0.29724180698394775, -1.7642337083816528, -1.6707780361175537, -0.7488911151885986, 0.18601004779338837, 0.6662970185279846, 0.8279224038124084, 0.2901657223701477, -0.4058447480201721, 0.5736209154129028, -0.19969512522220612, -0.6084197163581848, 0.7407466173171997, -0.5468960404396057, -0.1646084487438202, -1.0889582633972168, -0.42333683371543884, 0.5924564003944397, 0.04957389459013939, -1.8850752115249634, -0.3384115695953369, 0.0062121497467160225, 0.9346527457237244, 0.42695966362953186, -0.2514362633228302, 0.6334962844848633, -0.5864887833595276, -0.2612014412879944, 1.1018543243408203, 0.027586866170167923, -0.5882309079170227, -0.1933700293302536, 0.5080360174179077, -1.134413719177246, -0.2725466191768646, -1.213417410850525, -0.5096064209938049, -0.5658150315284729, 0.9865384101867676, -1.2617261409759521, -0.9590399861335754, 0.2969933748245239, -0.27595096826553345, 0.03840334713459015, -1.359850287437439, -0.009195158258080482, 1.6534367799758911, 1.1854115724563599, -0.14786681532859802, -1.5976964235305786, -0.8848827481269836, -0.46803879737854004, -0.4168643653392792, -0.04322925955057144, 0.20341147482395172, -0.16131174564361572, 0.515169084072113, -0.11478623747825623, 1.1958850622177124, 1.3138002157211304, -0.8080952167510986, -0.309246301651001, 0.10050181299448013, -0.5000606775283813, 0.536097526550293, -1.7238924503326416, -0.5871256589889526, -0.7243238091468811, 1.9362623691558838, 0.46342894434928894, -0.849189817905426, -0.5685617923736572, -0.14507658779621124, -0.5820567011833191, 0.9913670420646667, 0.606974720954895, 1.5061465501785278, 0.5875211954116821, -0.09835710376501083, -0.01752728410065174, 1.343841314315796, -0.08828729391098022, -0.06908000260591507, -0.10140559822320938, 0.5751843452453613, -0.9863795042037964, 1.3798259496688843, 0.8097729682922363, 0.6345091462135315, -0.047788891941308975, -0.2967362105846405, -0.5460866689682007, 0.25988897681236267, 0.8093539476394653, -0.739355206489563, -0.5767063498497009, 1.2998565435409546, 0.8552460074424744, -0.047970402985811234, -1.116108775138855, 0.5971953272819519, -0.2979479730129242, -0.5562591552734375, -1.2772817611694336, 0.8930861353874207, 0.33910664916038513, 1.4358946084976196, -1.5531965494155884], [0.4320383667945862, 0.5621662735939026, 1.378031611442566, -0.28268685936927795, 0.2015412151813507, -0.3636263906955719, -0.21013593673706055, 2.311917304992676, -0.8327676653862, -0.18384644389152527, -0.10504960268735886, -0.4230273365974426, 0.44342154264450073, 0.1172846108675003, -0.14201660454273224, -0.8444302678108215, 1.1597957611083984, -0.24328500032424927, 0.24696864187717438, -0.6333099603652954, -0.003212725045159459, -0.1955326944589615, 0.9459568858146667, 1.2056680917739868, 0.2921648919582367, -0.7037060856819153, 1.1756693124771118, 1.0453873872756958, -0.7364723682403564, 0.9149870276451111, -1.9081898927688599, 1.1592929363250732, 0.08779990673065186, 0.8406069278717041, 1.542480707168579, -2.3720619678497314, 0.7804309129714966, 0.9131630063056946, 1.208833932876587, -0.5112623572349548, 1.922176480293274, -0.355235755443573, 0.4070381820201874, 0.14286759495735168, 2.3685033321380615, -1.6294461488723755, 0.4398650527000427, 0.2971203923225403, 0.7288154363632202, -1.1091092824935913, -0.5437557697296143, -0.4768633246421814, 0.24705864489078522, 2.5770809650421143, -0.6276333332061768, 1.13680100440979, -0.6310626268386841, 1.281115174293518, 0.6128661036491394, 1.1445032358169556, -1.4193525314331055, -0.6830503344535828, -1.1345648765563965, 0.46637529134750366, -2.0115013122558594, -0.11028274893760681, 0.5548271536827087, -1.2769538164138794, -0.10713319480419159, 0.5171826481819153, -1.5523942708969116, -0.9743397831916809, 0.025090085342526436, -1.0736494064331055, 0.4703372120857239, 0.4897475838661194, 1.5264379978179932, -0.0618121400475502, 2.113107442855835, -0.8284077048301697, -1.232843041419983, 0.5999217629432678, -2.0659542083740234, -2.070657253265381, 1.4983781576156616, 3.2063822746276855, -1.0501707792282104, 2.1357085704803467, 0.4504515826702118, 0.8753564357757568, -0.03814847394824028, 2.3693058490753174, -0.24143221974372864, 0.6415765881538391, -0.9598867893218994, -1.1179131269454956, 0.8442159295082092, 1.964799165725708, 0.33167949318885803, -0.007075099274516106, -0.19185921549797058, -0.22458885610103607, -0.8593412637710571, 0.34763556718826294, 0.7562909722328186, -0.3661775588989258, 0.2697366178035736, 0.4590192437171936, -0.23204950988292694, 0.21491709351539612, 2.483829975128174, -0.770267128944397, -1.2532405853271484, 1.2193114757537842, 0.9338746070861816, -1.1222572326660156, 0.4425884783267975, 0.4928179681301117, -1.122369647026062, -0.9473024010658264, 2.1677134037017822, -0.7007459998130798, -0.6672590374946594, 0.9908799529075623, 0.8251497745513916, -0.8408181667327881, 0.9892977476119995, 1.0422906875610352, -0.2927020192146301, 0.9113745093345642, -0.8483126759529114, 0.3944834768772125, 0.7190032005310059, 0.23857057094573975, -1.253210425376892, -0.39130881428718567, -1.9403084516525269, -0.27357393503189087, 2.2307496070861816, 0.05794871225953102, -0.4694196879863739, 1.3643369674682617, -1.0328508615493774, 1.5900490283966064, -1.4540387392044067, 2.7443735599517822, 1.010905385017395, 1.0084532499313354, 0.6743066906929016, -0.2683355510234833, 0.22665080428123474, 0.2149876058101654, -0.6631174683570862, 1.5928466320037842, -0.696984589099884, 0.4526084065437317, -0.05154068022966385, -0.06259278208017349, -0.14043687283992767, -0.591946542263031, 0.7075071334838867, -1.011850357055664, 0.20405757427215576, -0.885394275188446, 1.0952204465866089, -0.6360313296318054, 0.7854481339454651, 0.6525143980979919, 1.0816106796264648, -0.18002645671367645, -1.3758286237716675, -1.460399866104126, 0.7427572011947632, -1.3714810609817505, -1.3946120738983154, -0.3846127390861511, 1.836489200592041, -0.7402858734130859, 2.4427757263183594, 0.48562854528427124, -0.5397883057594299, -0.6153167486190796, -1.4914861917495728, -0.18836522102355957, 0.7119842171669006, -0.9120035767555237, -0.34017956256866455, -0.7147101163864136, 0.09643539041280746, -1.2533661127090454, 1.4184463024139404, 0.7804611921310425, 0.24581697583198547, 1.4750583171844482, 0.03963550180196762, -0.5429385900497437, -0.2706085741519928, 0.33836886286735535, -0.6154115200042725, 0.7069525718688965, -0.24199460446834564, 1.246065378189087, 0.7728588581085205, -0.9231224060058594, -1.0705370903015137, -1.070716381072998, 1.3711249828338623, 1.1320239305496216, -0.6498033404350281, -1.2518543004989624, 0.9065379500389099, -0.6732052564620972, -0.4158112406730652, -0.9095902442932129, -0.4603935480117798, -0.7430415749549866, 0.927365779876709, 0.8061861991882324, -0.939850389957428, -0.022443823516368866, -0.9753273725509644, 1.9527363777160645, -0.18327787518501282, -0.784220814704895, 0.8958901762962341, 1.5330541133880615, -1.2267258167266846, -0.0452197901904583, 1.583276629447937, 1.8227782249450684, 0.6147888898849487, -1.1999528408050537, -0.38739699125289917, -1.3113014698028564, -0.976893424987793, 2.85040283203125, 1.2683464288711548, -1.3757028579711914, 0.035799283534288406, 1.334756851196289, -0.40950098633766174, 0.2631905972957611, 0.6506776213645935, -1.413733720779419, -0.04104374721646309, 0.5670000314712524, 0.22520506381988525, 0.6493869423866272, 2.3747198581695557, 0.28755640983581543, -0.34952083230018616, -0.9241114258766174, 0.10521818697452545, -0.2638187110424042, -1.5391827821731567, 0.7153421640396118, -1.2028898000717163, 2.1707136631011963, 0.9529829025268555, -1.337117075920105, -0.5600111484527588, 0.41084420680999756, 0.5095686316490173, -0.37538644671440125, 1.6163043975830078, 1.3724851608276367, -1.7268656492233276, -0.662290632724762, 0.35334813594818115, 0.8921866416931152, -0.7202184200286865, -0.2890113592147827, -0.06771345436573029, 0.014493737369775772, -0.7736160755157471, 0.9113513231277466, -1.1146262884140015, 0.3401293158531189, 0.1564241498708725, -0.20163613557815552, -0.5545552372932434, -1.0747485160827637, 0.011735293082892895, -1.19829523563385, -1.503265380859375, 0.8847190737724304, 0.6504135131835938, 0.08517573773860931, -0.49989083409309387, -0.06204914301633835, -1.220560908317566, -3.0718204975128174, 2.0765504837036133, 1.282651662826538, 0.08013773709535599, 0.9333863854408264, -1.1270591020584106, 2.210452079772949, -1.329106330871582, -0.027476221323013306, 0.0494685173034668, 0.38790690898895264, 0.7756797671318054, -0.37412360310554504, -0.7916653752326965, -0.8894622921943665, -0.3855174481868744, -0.7441445589065552, -0.1274803727865219, 1.1391814947128296, -1.6388615369796753, -0.014945344999432564, -0.1969999521970749, 0.4714162349700928, -1.2562733888626099, 0.6088110208511353, -1.4275707006454468, 0.5628422498703003, -0.9144675135612488, 1.2304655313491821, -1.259731650352478, -0.3251274824142456, 0.8953019380569458, -0.1501978188753128, 0.8030819296836853, 0.1141122579574585, -0.9529147148132324, 1.0148283243179321, -2.410912036895752, -0.6513768434524536, -0.054089341312646866, 0.44181469082832336, 1.3660080432891846, -0.6466548442840576, 0.34018802642822266, 0.5057494640350342, -0.3356306552886963, -0.9230730533599854, 0.6245182156562805, 0.3738543689250946, 1.2385166883468628, 0.060336314141750336, -0.42522943019866943, -1.6629189252853394, -0.5431162714958191, -0.1583024561405182, 0.20024505257606506, -0.13236382603645325, -1.3103704452514648, 1.102891445159912, -0.16972652077674866, 1.112738847732544, 0.0566646009683609, -0.37235891819000244, 0.009435154497623444, -0.33390548825263977, -0.15922142565250397, 0.4981621205806732, 0.3902685046195984, 0.5417912006378174, 0.5100152492523193, -0.9590277671813965, -0.62913578748703, -0.9859678745269775, -0.052091434597969055, 1.6234779357910156, -1.1741321086883545, 1.5787369012832642, -0.5888288021087646, -1.2325364351272583, -0.6251880526542664, 2.136404514312744, 0.4592481851577759, -0.609584391117096, -1.5723079442977905, -0.47633427381515503, -1.6490503549575806, 0.07203588634729385, -0.47144702076911926, -0.8048571348190308, 0.6272724270820618, 0.3656194806098938, -1.1882110834121704, 0.0784425213932991, -1.6789830923080444, -0.07911206781864166, -0.5792264342308044, -0.808128297328949, -0.1245453730225563, 0.10768967866897583, 0.7690204977989197, 0.9854124784469604, 0.1501992642879486, 0.4945127069950104, 2.265062093734741, 0.49695050716400146, -0.4657008945941925, -1.151361107826233, -0.14475120604038239, 0.23532623052597046, -0.3296026885509491, 1.620559573173523, 0.21076475083827972, 0.27250516414642334, 0.7484534978866577, 1.0341709852218628, -0.7374251484870911, -1.2168952226638794, 0.0956035703420639, 1.3932944536209106, 1.3032418489456177, 0.8898800015449524, -0.3827016353607178, 0.24300706386566162, 0.15462468564510345, 0.6040003299713135, 0.710480809211731, 0.9434604644775391, 0.059101447463035583, -0.8562445044517517, 0.8617194294929504, -0.773847758769989, 0.25625014305114746, 0.06048963591456413, 0.04307237267494202, -0.5965937972068787, 0.6373347043991089, 0.6768822073936462, -1.9566471576690674, -1.4803329706192017, 0.36026567220687866, 1.7467892169952393, 1.2241450548171997, 1.5078532695770264, 0.5138412117958069, 0.5568972229957581, -1.4634453058242798, 0.13922563195228577, 1.2671178579330444, 0.3264675736427307, 0.4713039696216583, -0.0825604796409607, 0.17446227371692657, -0.5989395380020142, 0.7267611622810364, -0.11203370243310928, -1.7238214015960693, 2.547459840774536, -0.6327241659164429, -0.6629045009613037, 1.2070587873458862, -0.6508901119232178, 1.1959173679351807, -0.7389686107635498, 1.3683102130889893, 2.6262094974517822, -0.9020993113517761, 1.3133962154388428, -0.23858638107776642, -0.9414108991622925, -1.6185296773910522, 0.5705931186676025, -0.26043447852134705, -0.4742448329925537, 0.6532513499259949, -1.4508041143417358, -1.6364127397537231, -0.4869808852672577, 0.0640479177236557, -0.21780773997306824, -0.10877487063407898, 0.8039110898971558, 0.2734590768814087, -0.4422703981399536, -1.002517819404602, -0.05103979632258415, 0.5045470595359802, -0.18478207290172577, -0.568792998790741, 0.17270906269550323, 0.11876414716243744, 0.6798590421676636, 0.18507397174835205, -0.3114875555038452, -1.080245018005371, 0.3449903130531311, -0.2130204141139984, -0.4699954092502594, 1.8698737621307373, -1.2148420810699463, -1.8672785758972168, -0.651797890663147, 0.23420986533164978, 0.8310273289680481, 1.174936056137085, 2.00624680519104, 0.2700366675853729, 0.3764573335647583, 0.8593993782997131, -0.31619831919670105, -1.0847989320755005, -0.7889564633369446, 0.9851223826408386, -0.1614929735660553, -0.7995122671127319, -0.03060968965291977, 1.3769289255142212, 0.23709779977798462, -1.2011559009552002, 0.34341946244239807, 0.005149688571691513, -1.11613929271698, -0.8223763704299927, 0.34459954500198364, -0.8146038055419922, -0.5519108772277832, 0.7039462327957153, 1.5469317436218262, 0.5396261215209961, 0.3869042694568634, -0.7854945659637451, -0.855154812335968, 0.26559555530548096, -0.27864813804626465, -0.2996567487716675, -0.1320342719554901, -0.40985220670700073, -0.023394715040922165, 1.073113203048706, 1.5917003154754639, 1.2753194570541382, 0.8514592051506042, 0.7800776958465576, -0.3896804451942444, 1.785448670387268, -0.8247326612472534, -0.7191304564476013, -0.0924459844827652, 0.4940893352031708, -1.21764075756073, -0.687553346157074, 0.7544561624526978, 0.9748111367225647, 1.6950843334197998, -0.9608392119407654, -0.0702148824930191, 0.6853331923484802, 0.9637780785560608, -0.5235652923583984, 0.7742412686347961, -1.3299185037612915, -0.6892831325531006, 1.4866578578948975, -0.8786716461181641, -0.7298015356063843, -0.27281057834625244, 1.3348979949951172, -0.0389978401362896, -1.6343581676483154, -0.6073533892631531, -0.02881907857954502, 0.697400689125061, 1.2410454750061035, 1.210665225982666, 4.042684555053711, -1.3580645322799683, -0.19391830265522003, 0.22453643381595612, -0.5136674642562866, 1.2616993188858032, 0.7786874771118164, -0.15069282054901123, 0.1091165542602539, -2.5326435565948486, 1.8045110702514648, -0.16576996445655823, -0.19213341176509857, -0.2680533528327942, 0.022547336295247078, -1.24054753780365, 0.5461240410804749, 1.285609483718872, 1.255624532699585, 0.18706883490085602, 0.6226030588150024, -1.8609578609466553, -2.5094892978668213, 0.8052429556846619, -0.051117584109306335, -0.21716174483299255, -2.1065475940704346, -1.3152257204055786, -0.16982616484165192, 0.028373822569847107, 1.1667941808700562, 1.3066438436508179, 0.24644683301448822, 0.85072261095047, 1.358013391494751, -0.3762087821960449, -1.1965157985687256, 0.0409720204770565, -1.6250920295715332, -1.8037983179092407, -0.9184228181838989, -0.5186936855316162, 0.5996770858764648, -0.28111350536346436, 0.16694976389408112, -0.2240365892648697, -0.16832558810710907, 0.4931907653808594, 1.868757963180542, -0.749809980392456, 0.5843285322189331, 0.14474132657051086, 2.2809321880340576, 0.2791513502597809, -0.1128760576248169, 0.030304288491606712, -0.050346534699201584, 0.2901165187358856, -2.0067665576934814, 0.1457531750202179, -1.5592485666275024, -0.8661335706710815, 0.6240445971488953, 0.8683916926383972, 0.052847810089588165, -0.5474656224250793, 1.1891546249389648, -0.6205562949180603, -1.285706639289856, 0.6197642087936401, 1.5353690385818481, 1.5306073427200317, 0.44742998480796814, 2.624483823776245, 0.39182767271995544, -0.010758582502603531, 0.8647521138191223, 0.7071548104286194, -0.6389545798301697, -1.2485520839691162, -1.3517729043960571, -1.1943237781524658, 0.5132678747177124, 0.968613862991333, 0.32657137513160706, -0.41494524478912354, -1.8506956100463867, -0.47127166390419006, -1.5792236328125, -1.057551622390747, 1.4479186534881592, -0.8720929026603699, -0.7547817826271057, 1.0211105346679688, 3.3131909370422363, 0.8286551237106323, 0.6913599371910095, -0.13289986550807953, 0.3763754367828369, -1.8913005590438843, -0.10179881006479263, 0.6099249720573425, 1.1383495330810547, -1.0330709218978882, -0.37287527322769165, -2.0459814071655273, -0.7176880836486816, -1.143021583557129, 1.0773988962173462, 0.5084774494171143, -0.5569174885749817, 0.7992744445800781, -1.2329559326171875, -0.610605001449585, -0.8206501603126526, -0.23886942863464355, -0.9039813280105591, -0.6113573312759399, 0.9315957427024841, 1.5531795024871826, 0.5429759621620178, -0.5875352621078491, -0.5717455148696899, -0.10676313936710358, 1.8424545526504517, 0.24880380928516388, -0.32142922282218933, 0.10807205736637115, 0.6040006875991821, 1.0538262128829956, 0.19520069658756256, 0.84787517786026, 1.879723072052002, 0.04524713754653931, 0.6892676949501038, -0.004769483115524054, 0.20858976244926453, 1.900227427482605, -0.8326233625411987, -2.1381192207336426, 1.4811002016067505, -0.3682040870189667, 0.2566740810871124, -0.8824334740638733, -0.7049123048782349, -1.5247480869293213, -0.2024713009595871, -0.3008198142051697, -0.8363656997680664, -0.29963207244873047, 0.4906034469604492, -2.069410562515259, 1.2051138877868652, 0.6516540050506592, -0.5865132212638855, -0.9390265345573425, 0.6819353699684143, 0.053367987275123596, 1.6285345554351807, -0.15731970965862274, 0.04387577250599861, -1.0143836736679077, -0.0327865406870842, -1.2676737308502197, 0.24823825061321259, 1.215163230895996, -0.1019500344991684, -0.28930458426475525, 2.718658447265625, -0.44707322120666504, 2.2449893951416016, 0.3130100965499878, 0.4412437677383423, 1.3409062623977661, 0.07520542293787003, 0.31512022018432617, 2.025991916656494, 1.3630913496017456, -0.7483750581741333, -0.07137089967727661, -1.618036150932312, 3.437786817550659, 0.16633832454681396, 0.16859936714172363, -1.3447623252868652, 1.1168203353881836, 1.0158376693725586, 0.9174541234970093, 0.3524835705757141, 2.229557991027832, 0.20697540044784546, 1.1796015501022339, -0.5219709873199463, 1.1243107318878174, 2.520801544189453, 0.6511126756668091, 1.5808978080749512, 1.639133095741272, 0.8223702907562256, -0.9409273266792297, -0.3819005787372589, 2.7126646041870117, 0.4242194890975952, 1.1154142618179321, 0.4171667993068695, 0.10299675911664963, 1.2512918710708618, 0.8916584253311157, -1.5349352359771729, -3.5032434463500977, 0.7086498737335205, 1.770019769668579, -0.45635131001472473, 1.1490452289581299, -0.7706122994422913, 1.1332436800003052, -0.15760816633701324, -0.13879643380641937, 0.6039337515830994, 0.137873575091362, -0.28609907627105713, -0.49781301617622375, -0.10114815086126328, -0.6911342740058899, 0.9059426784515381, -0.12723718583583832, 1.5843493938446045, -0.5852643847465515, 0.8513157367706299, 0.5778462290763855, 0.5721693634986877, -1.2509448528289795, -0.36787888407707214, 0.19325131177902222, -0.5679042935371399, 0.13982194662094116, -0.9073348045349121, 0.0378744900226593, 0.7668170928955078, 0.21800361573696136, 1.6719467639923096, -0.7078945636749268, 1.0342235565185547, 0.6656742095947266, -0.41707536578178406, 1.6361923217773438, 0.3057332932949066, 1.495252013206482, -0.5576359033584595, 0.46998703479766846, 1.3654780387878418, 0.21885491907596588, -0.02082742750644684, 0.5822349190711975, 1.230870008468628, 2.086303949356079, -0.07383271306753159, -0.35475778579711914, -2.06620192527771, 0.4503069519996643, 1.1502846479415894, -0.15658248960971832, 1.2325633764266968, -0.4149990379810333, -0.4133021831512451, -0.15773606300354004, 0.9218788146972656, 1.5102990865707397, -1.4875819683074951, -0.5067710876464844, -1.3130854368209839, 0.11987806111574173, -0.06883454322814941, -0.45210564136505127, -0.7505954504013062, 0.7604617476463318, 0.22685740888118744, 1.1581166982650757, 0.029454734176397324, -0.8439120054244995, 0.30118346214294434, 0.10539623349905014, -1.14155912399292, -0.043811947107315063, 0.9613481760025024, 0.7671879529953003, 0.44277051091194153, -0.22689266502857208, 0.055679768323898315, -1.7185965776443481, 0.05995194986462593, -0.7814964652061462, -0.24685952067375183, 1.0449224710464478, 1.1542185544967651, -0.8768737316131592, 0.18923118710517883, -2.443882465362549, -1.420456051826477, 0.46607261896133423, -0.172660231590271, 1.2203161716461182, 0.6843768954277039, -0.6963779926300049, 1.7170909643173218, -1.391163945198059, 0.7667562961578369, -0.02797481045126915, 0.6940632462501526, -0.434970885515213, 0.5972037315368652, -1.755620002746582, -0.07401703298091888, 2.4457900524139404, 0.23432809114456177, 0.729238748550415, -0.4781258702278137, 1.4376405477523804, -0.1739601492881775, -0.3597927987575531, 0.01781967654824257, -0.4551517963409424, 0.49443304538726807, 0.9651252031326294, 0.9804373979568481, -0.3350697457790375, -1.0599206686019897, -0.21540778875350952, -0.0286000557243824, 0.26636332273483276, 0.7657936811447144, -0.35898616909980774, 0.8198084235191345, 0.7057576775550842, -0.9806522727012634, -1.642107605934143, 2.0633811950683594, -0.5924711227416992, 0.8685516715049744, 1.097821831703186, -0.11827235668897629, 1.853849172592163, 0.2946344316005707, 0.5326501131057739, -1.2934799194335938, 0.9113550782203674, -1.2317306995391846, -1.2246607542037964, -0.8016741275787354, -1.466497540473938, 1.588233470916748, -1.2064076662063599, -1.167519450187683, -1.7988697290420532, 0.39537519216537476, 0.5919867157936096, 0.4421655833721161, 0.18033970892429352, 1.158384084701538, -0.7143144607543945, -1.7689982652664185, 1.090088963508606, 0.9239780306816101, 1.3495591878890991, -0.653045117855072, 1.307101845741272, -0.5948264598846436, 0.8800243735313416, -1.7930965423583984, -0.30067381262779236, 0.18143345415592194, -0.10363899916410446, -1.195337176322937, -0.5304129719734192, -0.13772553205490112, 1.1448460817337036, -0.2606121599674225, -0.601790189743042, -0.35759836435317993, 0.07358890771865845, -1.3304553031921387, -0.30125728249549866, 0.9394440650939941, -0.8232027888298035, -0.29483264684677124, 1.4024536609649658, 1.7951573133468628, -1.3306384086608887, 1.3128535747528076, 0.7427797317504883, -0.7193684577941895, 0.3446372151374817, -0.9034433364868164, -1.4734140634536743, 0.4283529818058014, 0.6293149590492249, 0.8232297301292419, 0.32889431715011597, -0.2561911344528198, 1.7757750749588013, -0.4393565356731415, 0.3477467894554138, -2.313732862472534, -0.41381365060806274, 0.11798442155122757, 0.8673239350318909, -0.09350433945655823, 0.6824076175689697, -1.5172888040542603, 1.1518911123275757, -0.6819656491279602, -1.1358290910720825, -0.6805378794670105, 1.1034135818481445, 0.9177780747413635, -0.00935762282460928, -0.18480324745178223, 0.8520300388336182, -0.3678905963897705, -0.5376262068748474, -0.9587112069129944, 0.4470509886741638, 0.640794575214386, -1.4150645732879639, 1.3759145736694336, -0.3345126807689667, -0.47200605273246765, -0.049119435250759125, -0.7955753803253174, 0.3550666272640228, -0.2562800347805023, -1.420837163925171, 0.8164933323860168, -1.3913112878799438, 1.2564823627471924, -1.2749454975128174, -0.09093553572893143, -1.2748280763626099, 2.267455577850342, -1.2696309089660645, 0.13456201553344727, 0.3392126262187958, -0.38070356845855713, -0.5076270699501038, 1.0825973749160767, 0.3860170245170593, -0.7187567353248596, -0.4314982295036316, -0.5221009850502014, -0.6199513673782349, -0.5801753401756287, 0.4419024884700775, -0.7578074336051941, 1.6363471746444702, 0.4040031135082245, 0.5588566660881042, 1.1501041650772095, -0.28205397725105286, 0.7398145794868469, -0.6603845953941345], [-0.28032419085502625, -0.4418504238128662, 1.9718644618988037, -0.8256560564041138, 1.734886646270752, 0.4240070879459381, -1.486490249633789, 1.4916263818740845, -0.12979011237621307, -0.35337814688682556, 0.8159148097038269, -0.7583443522453308, 0.24172210693359375, 1.5662014484405518, -0.2641148865222931, 0.3880828022956848, -2.6039652824401855, -1.1079185009002686, 0.5713241100311279, -1.3063063621520996, 0.6776711344718933, 0.7752727270126343, -0.7994511127471924, 1.5297425985336304, -2.547569751739502, 0.18230143189430237, 1.569508671760559, -0.4287048876285553, -0.16105051338672638, 0.10092703998088837, -2.002617835998535, 0.4488059878349304, -0.181793212890625, 0.14276857674121857, 1.1478047370910645, -0.831836462020874, -0.11159560084342957, 0.4960240125656128, -1.9751887321472168, -0.032813698053359985, 0.10235164314508438, -0.63426274061203, 1.6811493635177612, -2.113719940185547, -0.1317012906074524, 0.6535260677337646, 1.0668481588363647, 0.2813352048397064, 0.8625929951667786, 1.2333590984344482, -1.0438745021820068, -1.0460236072540283, 1.2471272945404053, -0.9866587519645691, 0.3125421702861786, 2.349797248840332, -0.8955973982810974, -1.5786644220352173, -0.06699739396572113, -1.2484875917434692, 0.23840190470218658, 1.7400939464569092, -1.6113240718841553, -0.2529990077018738, 1.021328330039978, -2.1399483680725098, 0.3867262005805969, 1.8431224822998047, -0.3190329968929291, -0.2577800154685974, -0.3932221829891205, -1.162520170211792, -0.08915097266435623, 1.8495423793792725, 0.42206817865371704, -0.6274502277374268, -1.0571058988571167, 0.9491283297538757, -0.05544998124241829, -0.7914038300514221, 0.6130326390266418, -0.0900198295712471, -0.5015968680381775, -1.9629729986190796, -0.5476149916648865, -0.820429801940918, 1.3754932880401611, -1.1107957363128662, -0.1727023720741272, -1.3700610399246216, -1.1859586238861084, -0.17350569367408752, -0.24845696985721588, -0.2283034771680832, -0.4380200505256653, -0.3770689070224762, 2.1585919857025146, 0.6906813383102417, -0.336856484413147, 0.2921878695487976, 0.6898893117904663, 0.9383988976478577, -0.07803496718406677, -0.20880693197250366, 0.7953289747238159, -0.9355932474136353, -1.247650146484375, 0.12952736020088196, 1.4579575061798096, 0.24928230047225952, -0.8697829246520996, -0.7969940900802612, -0.4780765771865845, 0.32341405749320984, 0.7348872423171997, 0.09266360104084015, -0.9143832921981812, 1.5022963285446167, 1.8954437971115112, 1.3807587623596191, 0.16813473403453827, 2.314610004425049, 1.3374614715576172, 0.059771161526441574, -1.0500870943069458, 0.9523054957389832, 0.7137855887413025, 0.6982897520065308, -0.6786044836044312, -1.4524307250976562, -0.5420328378677368, 0.1433308869600296, -0.31317391991615295, 0.43056660890579224, 0.8461124300956726, -0.7956462502479553, 0.6292922496795654, 0.4842449724674225, 1.3465524911880493, -0.03343217819929123, -0.5532017946243286, 0.6877495050430298, -1.438066840171814, 0.6198500990867615, -0.9304569959640503, 0.28992611169815063, -0.9441214799880981, 0.11293420940637589, -0.46105116605758667, 1.1452765464782715, -0.9437344074249268, -0.04803290590643883, -1.5176050662994385, -2.0295979976654053, -0.23789359629154205, 0.9503248333930969, 1.7100212574005127, -1.898740291595459, 2.0115575790405273, -0.13266344368457794, -0.9026111364364624, 0.7022849321365356, -0.9411993622779846, 0.396537184715271, -0.09084563702344894, 0.3996548652648926, 0.0025895528960973024, 1.4609770774841309, -0.14138104021549225, 0.8688259124755859, -0.3455705940723419, -1.0139882564544678, -0.179799422621727, -1.2567734718322754, 1.2248526811599731, 0.4348139762878418, 1.5250561237335205, 1.220741868019104, 0.7060109376907349, -0.3327599763870239, 0.14122964441776276, 2.0813493728637695, -1.3907502889633179, -1.2925488948822021, -1.0026594400405884, 0.23488767445087433, -1.3032599687576294, 0.2391689419746399, -0.46174871921539307, 0.36087414622306824, -0.5886951088905334, -1.2729383707046509, -0.9913100004196167, -0.16403089463710785, -1.2676661014556885, 0.09216075390577316, 0.12159986793994904, 1.1337422132492065, -1.158997893333435, 0.35735639929771423, 1.083707332611084, 0.22856171429157257, -0.2561893165111542, -1.2113128900527954, -1.828231692314148, 1.0954736471176147, 0.5987381935119629, -0.9927175641059875, -2.158331871032715, 2.557461977005005, -1.7864199876785278, 0.8521664142608643, -0.3647944927215576, -0.01975916139781475, 0.033341627568006516, 0.2909282147884369, 1.4603573083877563, 0.17921128869056702, 0.6359387040138245, -0.7185482382774353, 1.2223623991012573, -0.06240365654230118, 0.28188368678092957, -1.740346074104309, 1.265668272972107, 0.6913745999336243, 0.11870229244232178, -0.6192427277565002, 0.8469072580337524, -0.2721964120864868, 1.0700644254684448, -1.4508821964263916, -1.164276361465454, -0.12558351457118988, -0.2542644441127777, 0.47528889775276184, 0.967067301273346, -0.26758652925491333, 0.6564750671386719, -0.751670777797699, 1.9699962139129639, 0.5626221895217896, 0.22251224517822266, -0.0015840063570067286, 0.29952171444892883, -0.060500066727399826, -1.2166223526000977, 0.039018142968416214, 0.08397328853607178, -0.0715605765581131, 0.31766200065612793, 0.684280276298523, 0.07700924575328827, 0.8343374133110046, -0.41976866126060486, 0.016376806423068047, 0.964093029499054, 1.7786729335784912, 0.01874368265271187, -0.8195956945419312, -0.2242487072944641, 1.428128957748413, -0.9685172438621521, 0.38516947627067566, -0.4993119239807129, 0.39492565393447876, 1.4024745225906372, 0.529202938079834, 0.6256487369537354, -0.982846736907959, 1.2361119985580444, 1.1930345296859741, -1.9967328310012817, -0.717728853225708, -0.35299205780029297, -0.9253958463668823, -0.3331719934940338, 0.19441823661327362, -0.27062085270881653, -0.8888404369354248, -0.04942058026790619, -0.9864298701286316, -1.1016069650650024, 0.562906801700592, -0.9508771300315857, 0.764564573764801, 0.3062479496002197, 0.45480793714523315, 0.5131838917732239, 0.6200975179672241, -1.0421446561813354, -0.8242339491844177, 1.338771104812622, -0.1263815313577652, 1.1893724203109741, -0.08103355020284653, -1.3184006214141846, -0.6126061677932739, 0.5657325387001038, 0.3582323491573334, 1.6364341974258423, -0.2608645558357239, 0.9964193105697632, -0.34879404306411743, -0.16508901119232178, 0.6962466239929199, 1.0094280242919922, 1.2463431358337402, -0.7568221688270569, -0.783644437789917, -0.9026007056236267, 1.562949299812317, 0.2060462385416031, 1.1299155950546265, 0.33480703830718994, 0.7334527373313904, 0.6120627522468567, -1.354251503944397, -0.9093246459960938, -1.8077255487442017, -1.394707441329956, 0.58786541223526, -1.4432904720306396, 0.2923746109008789, -1.2294071912765503, 0.6318714022636414, -0.31560462713241577, 0.39040738344192505, -0.16807885468006134, 1.6463443040847778, -0.8588606715202332, 1.238073468208313, 0.559637725353241, 1.9974725246429443, -0.9623908400535583, -0.34248092770576477, -1.5912457704544067, 0.7950674891471863, 0.5464974045753479, -0.9167088866233826, 0.5198545455932617, 0.7985261678695679, 0.20857831835746765, -0.4151487648487091, 0.024425573647022247, -1.7599995136260986, 0.027138765901327133, -0.6373283267021179, 0.9743145704269409, -0.3644183576107025, -0.11442456394433975, -0.7209128141403198, 0.39697572588920593, 0.7952703833580017, 0.2745639979839325, -1.0300713777542114, -0.14238077402114868, -0.30855560302734375, 1.0942201614379883, -0.760345995426178, -1.8932311534881592, -0.47328680753707886, 0.0085932407528162, -0.11078935116529465, 0.5413358211517334, 0.7369885444641113, -0.9175817966461182, 1.2494555711746216, -0.1795596182346344, -0.6207271814346313, -0.39444607496261597, -1.8292031288146973, 0.5263915657997131, 1.0225971937179565, -1.2428505420684814, 0.05869122967123985, -0.030840205028653145, 0.5829149484634399, -0.977116048336029, 1.4009184837341309, 0.834959089756012, -0.42566415667533875, -1.701330304145813, -1.845462679862976, 0.8902956247329712, 0.8826161026954651, -0.1155020222067833, 1.3412514925003052, -0.3426530063152313, 0.20799270272254944, 0.917102038860321, 0.4511094093322754, 0.3283956050872803, 0.7661654353141785, -1.1002397537231445, 0.6866683959960938, -1.1628984212875366, -2.9240562915802, -1.8522282838821411, -0.1289730668067932, 0.9664655923843384, 0.5064761638641357, -0.7208395600318909, -0.23565609753131866, 0.01600758358836174, -1.8117088079452515, 1.799970269203186, 1.2924047708511353, 0.5343188643455505, -0.3490425646305084, -0.09672015905380249, 1.6339305639266968, 0.48820510506629944, -0.08380786329507828, 0.31240198016166687, -0.7771877646446228, -2.49763560295105, 0.24543194472789764, 0.5497772693634033, 0.3291681706905365, -1.6504693031311035, -2.0565547943115234, 0.9020509123802185, 0.398045152425766, 1.8818094730377197, -0.025817178189754486, 0.8557222485542297, 0.46094539761543274, 1.2688465118408203, -0.32208627462387085, 1.4045815467834473, 0.6669901609420776, 0.5427602529525757, -0.8027318716049194, -0.3079797923564911, -1.004223346710205, 0.7451003789901733, -1.7675132751464844, 0.27390971779823303, -0.7840815782546997, -1.3415191173553467, 1.4605904817581177, -1.3708863258361816, -1.9158838987350464, 0.734910786151886, 0.7136144042015076, -0.7945759296417236, -0.40881937742233276, 0.9894951581954956, 1.8820648193359375, 1.204347848892212, 0.5265007019042969, -0.2153053879737854, 0.3137735426425934, 0.6103101968765259, 1.5120232105255127, -0.1718394011259079, -0.1676732748746872, 0.3947216272354126, -2.183342218399048, 0.4172992408275604, 0.2146511971950531, -0.13532093167304993, 0.2088424265384674, 0.10801243036985397, 1.2467480897903442, 0.43714362382888794, -1.3178189992904663, 0.4810350239276886, -1.5286222696304321, -0.043787579983472824, 0.19755202531814575, 0.39061984419822693, -0.16620250046253204, -0.7502005696296692, 0.5832256078720093, 0.8721447587013245, 1.7724815607070923, -0.5334227085113525, -1.3767859935760498, -0.16994717717170715, -1.610817551612854, 2.9040188789367676, 1.1585335731506348, -1.321847677230835, -0.8879566788673401, 1.0204252004623413, 0.3177797496318817, 1.1978715658187866, -0.9990361332893372, -1.26860511302948, 0.6104788780212402, 0.015006772242486477, 0.9463030695915222, -1.1387183666229248, 0.3732144236564636, 0.1143898144364357, -2.5572032928466797, 0.7880109548568726, 0.8266369104385376, 0.8190976977348328, 1.1012678146362305, 0.8802257776260376, -0.2794629633426666, -0.29796335101127625, -0.11354652792215347, 0.7007931470870972, -1.1409838199615479, 1.7885961532592773, 0.7642402648925781, 1.639661192893982, 0.6378332376480103, -0.3278406262397766, -1.0738343000411987, 2.5202691555023193, -0.7433440685272217, 0.9705166220664978, 1.0193244218826294, 2.040898561477661, -1.6195638179779053, -1.4991908073425293, -0.042627062648534775, 0.40102463960647583, -1.1619981527328491, 0.050228532403707504, 0.6332237720489502, -0.753591001033783, 0.2401776760816574, 0.6724150776863098, 1.230378270149231, -0.22464153170585632, 1.004414439201355, -0.292299747467041, -0.010778249241411686, 0.12279029190540314, -0.49018263816833496, -0.9632886052131653, 0.3378463685512543, 0.15964873135089874, 0.36102476716041565, 1.0666142702102661, 1.1649922132492065, 2.8109235763549805, 0.016628729179501534, 0.322209894657135, 0.9820472598075867, 1.4596779346466064, -0.20743094384670258, 1.1971931457519531, -2.7191884517669678, 0.744279146194458, -0.6619100570678711, 0.6906960010528564, -1.0114414691925049, 0.7382413744926453, -1.0070427656173706, 2.077824354171753, -0.3029312491416931, -0.6760542392730713, -0.0968400090932846, -0.6087489128112793, 1.290659785270691, 0.8595766425132751, 0.4026845693588257, -1.7251397371292114, -0.9281243681907654, -0.09807180613279343, 0.44975680112838745, -0.1676516830921173, -1.3455613851547241, 0.9938806891441345, 0.5772557854652405, 0.944716215133667, 0.5029146075248718, 0.42670732736587524, 1.4960559606552124, -0.5727675557136536, 0.9997820258140564, -0.3842058479785919, 1.3145865201950073, -1.3250950574874878, 0.21330267190933228, 0.3464322090148926, 1.0620498657226562, -0.001160479267127812, 0.0828339084982872, 0.3650834858417511, -0.5698311924934387, 0.9693857431411743, -0.3976224660873413, 0.655061662197113, -1.515072226524353, 0.5495367646217346, -1.3393974304199219, -1.5070966482162476, -0.04763093590736389, -1.7107335329055786, -1.1436156034469604, 2.897907018661499, 0.21341796219348907, -0.8430456519126892, -0.43768206238746643, -0.9580011963844299, 1.2669488191604614, -1.377040982246399, -2.1352572441101074, -1.5147573947906494, 0.20298567414283752, 0.5982080101966858, 0.014567422680556774, 0.07376054674386978, 0.44776034355163574, 0.27481168508529663, -0.40529271960258484, 0.8078834414482117, -1.6793392896652222, 0.350693941116333, 2.246586561203003, -1.916298747062683, -0.14684973657131195, -0.08232393115758896, 2.9889864921569824, 2.0994560718536377, 2.437730073928833, -1.0056568384170532, -0.5430881977081299, 0.060912419110536575, 1.674034595489502, 0.9507257342338562, -0.641579270362854, -1.310575246810913, -0.23293665051460266, 1.0631743669509888, 0.22476010024547577, -0.06007213890552521, 0.18669366836547852, -1.3591618537902832, -1.1288028955459595, -1.3896467685699463, 0.5678713321685791, -0.42771875858306885, 1.0267232656478882, 0.3899116814136505, -0.38954660296440125, 0.7878680229187012, 1.9299554824829102, -2.201986789703369, -0.9923685789108276, 1.3941407203674316, 0.296729177236557, 1.7967579364776611, 0.962207019329071, -0.7026965618133545, -0.09562232345342636, 0.3336806297302246, 2.350614070892334, 0.4280826151371002, -1.164873719215393, 0.8509843945503235, -0.4825778007507324, 0.6371143460273743, -1.102933645248413, -0.40273788571357727, -0.6180562376976013, -1.7504740953445435, 0.6919811964035034, 1.8296000957489014, -1.1477826833724976, 0.705358624458313, 1.2528120279312134, -0.3983875513076782, -0.3503723740577698, -0.9902521371841431, 0.31933438777923584, 0.33679458498954773, 0.6503248810768127, 0.2418610155582428, -2.3982505798339844, 1.1562097072601318, 1.1001288890838623, 0.13971757888793945, -1.2093950510025024, 1.2756038904190063, -2.2552919387817383, 0.12468668818473816, 0.6167207956314087, 0.7819132208824158, 0.49577775597572327, 1.2425289154052734, -0.7711836695671082, -0.03802599385380745, 1.0139228105545044, -0.7682148814201355, 0.8069265484809875, -0.32272815704345703, -0.8433132171630859, 0.3596695065498352, 0.16176439821720123, -0.9874488115310669, -1.5256214141845703, 0.8291835188865662, -1.361562728881836, 0.882114589214325, 0.24648821353912354, -0.6045146584510803, 0.51386958360672, -0.8289762735366821, 1.6587830781936646, -1.9226146936416626, 0.29687169194221497, 2.097898006439209, -0.7976223230361938, -0.9550584554672241, -1.3536874055862427, -0.4594370722770691, 0.2694266140460968, -0.6406599879264832, 0.6640684008598328, -0.26413872838020325, -0.5008750557899475, -1.142201542854309, 1.4385144710540771, -1.5419789552688599, 0.38333365321159363, -1.4786694049835205, 0.5783323049545288, -2.521350622177124, 1.581613302230835, -0.6127409934997559, -0.48445430397987366, -0.3393271863460541, 0.3070300221443176, 0.32901743054389954, -0.21410995721817017, -0.10306871682405472, -0.620845377445221, -0.2023182213306427, 0.42594873905181885, 0.6377304792404175, 0.03848829120397568, 0.6206473112106323, -0.22613447904586792, 0.20211191475391388, -0.160095676779747, 0.046160731464624405, -0.7059372663497925, -0.6863563656806946, -0.09225012362003326, -0.1371370553970337, -1.532594919204712, 0.5581265687942505, -1.2283203601837158, -1.2692792415618896, -0.28783443570137024, -1.1313141584396362, -0.9664728045463562, 0.8187118768692017, 0.1880963295698166, -0.14391890168190002, -1.055126667022705, -0.3952828645706177, -2.123236894607544, 1.0636945962905884, -0.024647843092679977, -0.5232122540473938, -1.608394980430603, -0.4583393335342407, 0.5984981656074524, -0.186717689037323, -0.4339381158351898, -1.303362488746643, 0.015580791980028152, 0.7090873718261719, 0.7993648052215576, -0.35955753922462463, 1.2268447875976562, 0.12075432389974594, -1.4639227390289307, -0.7259563207626343, 1.2610446214675903, 0.56378573179245, -0.2949281930923462, -0.13943341374397278, -0.49990183115005493, -1.0543386936187744, -1.0760797262191772, 1.1772785186767578, -0.3938629925251007, -1.1852933168411255, 0.0734948068857193, 0.39819055795669556, -0.38637039065361023, 1.6614032983779907, -0.42783668637275696, 0.46130919456481934, 0.24017047882080078, 0.5414737462997437, 0.6641066074371338, 0.5129932165145874, -0.36121779680252075, 0.48741066455841064, 0.4847036302089691, 1.3330634832382202, 1.2279481887817383, 0.1527756154537201, -0.4418182373046875, 1.1636807918548584, 0.6905066967010498, -0.6581565141677856, -0.23563507199287415, -0.5889483690261841, -1.195921778678894, -0.160638228058815, 1.0180479288101196, 1.3823970556259155, 1.7299822568893433, -1.083612084388733, 2.21722412109375, 0.5131067633628845, -0.03989879786968231, 1.0459884405136108, 2.3660659790039062, -0.44659486413002014, -1.3999223709106445, -0.1443602293729782, -1.313561201095581, 0.7088178396224976, 2.9457080364227295, -0.636981189250946, -0.19660703837871552, -0.2285711020231247, -1.1151454448699951, 0.007437050808221102, -0.5938402414321899, 1.2652215957641602, 0.6756276488304138, -0.3647376596927643, 0.921109139919281, 0.7060831189155579, 0.227082297205925, 0.8185201287269592, -2.238585948944092, 0.3573175072669983, 0.41975948214530945, 1.3378281593322754, -0.7762038707733154, 0.6940587759017944, -0.42303600907325745, 0.567973256111145, 0.6227661967277527, -0.4469999670982361, -0.9247113466262817, 0.2842508554458618, 0.08429751545190811, 0.9013634324073792, 1.7856637239456177, 0.30469822883605957, 1.4151556491851807, 0.3179791271686554, 0.9443985223770142, 1.1009440422058105, -1.3898190259933472, -0.811980664730072, -0.8374114632606506, -0.6522367000579834, 1.8167552947998047, 0.8656821250915527, -0.7290929555892944, -0.14436876773834229, 0.7606102824211121, 0.709895670413971, -0.5087542533874512, 0.8533939123153687, -0.4703223407268524, -0.9536445736885071, -1.655704140663147, -2.1786530017852783, -2.2083160877227783, -1.1572725772857666, 0.33210045099258423, 1.0855937004089355, 0.5315709710121155, 1.1822926998138428, 0.5438976287841797, 0.1410064846277237, -0.17771051824092865, -0.33928507566452026, -1.4828420877456665, 1.1140170097351074, -0.039128780364990234, -0.1687157303094864, -0.6891860961914062, -0.14127863943576813, 1.5995709896087646, -1.3030171394348145, -0.1897866278886795, -0.22242100536823273, -0.5877675414085388, 1.1890524625778198, 0.5102062821388245, -1.5273385047912598, -0.5318393707275391, -0.29371166229248047, 1.153558373451233, -1.141858696937561, -1.0666487216949463, 0.27311286330223083, -0.007519796956330538, -0.9279381036758423, -0.781843900680542, 0.0905519425868988, -1.006731390953064, 0.7810570597648621, 0.8503967523574829, 0.7712858319282532, 0.7930868864059448, -0.41886284947395325, -0.8544191718101501, 0.8193543553352356, -0.7022485733032227, -1.0215085744857788, -0.06649619340896606, 0.2564924359321594, -0.5376991629600525, 0.48486438393592834, -1.0293715000152588, 0.5006836652755737, 0.42223599553108215, -0.14001117646694183, -1.5962728261947632, 2.095482587814331, -0.3273586630821228, 0.30236321687698364, 1.2043601274490356, -1.6862977743148804, 0.38826656341552734, 1.1185483932495117, -0.5222588181495667, -0.07151611894369125, -0.21522074937820435, -1.1793166399002075, -1.9534049034118652, -0.027413515374064445, -0.15495237708091736, -0.9954075217247009, -0.7891156077384949, -0.957780659198761, 0.7786884307861328, 0.4849312901496887, 0.15383023023605347, -1.2661716938018799, 0.667759895324707, 0.5463730096817017, 0.18657508492469788, 0.6817715167999268, 0.7609173059463501, -0.5337189435958862, 0.6990052461624146, -1.3451943397521973, 0.704070508480072, 1.9031977653503418, -0.8843877911567688, 0.42938026785850525, -1.7615375518798828, -2.3876583576202393, 1.1653982400894165, 1.0191701650619507, 0.6294452548027039, 0.27220404148101807, 1.2001560926437378, -0.4233013391494751, -0.8893141150474548, 0.6003629565238953, 1.340857982635498, 0.03024560958147049, 0.6850759983062744, -0.9725944995880127, -0.2652778923511505, -0.8148631453514099, -0.7732665538787842, -0.634290874004364, -1.1236274242401123, 1.1047364473342896, -1.4392869472503662, -1.2335333824157715, -0.14698591828346252, -0.5656606554985046, 1.067755937576294, -0.1040034145116806, -1.627448320388794, 1.843299388885498, -2.558030843734741, 0.011220685206353664, 1.3438067436218262, -0.3562018871307373, -1.486669898033142, -0.08316293358802795, -0.009486865252256393, 0.5828636288642883, -0.830032229423523, 1.0081753730773926, 0.583759069442749, 0.09155501425266266, 0.1331605464220047, 1.1033945083618164, -0.9944256544113159, -0.8947471976280212, 0.38258102536201477, -1.225858449935913, -0.10933353006839752, 0.7816165685653687, 0.5313209295272827, -0.16778947412967682, -0.9475094079971313, -0.010375037789344788, 0.40456870198249817, 1.0518848896026611, -0.03558052331209183, 0.836367666721344, 1.8962879180908203, -0.7681032419204712, -2.2411816120147705, -0.11792334914207458, -1.7039741277694702], [0.6066992282867432, -0.8161554336547852, 1.373221516609192, 0.6629970073699951, 0.1590072363615036, 1.2592943906784058, 0.4109238386154175, -0.0028987552504986525, -0.3610460162162781, 0.4244753420352936, -1.3464678525924683, 0.12755101919174194, 1.4802017211914062, -1.0237780809402466, -0.8812857866287231, 1.5084854364395142, -1.3685482740402222, -0.442985862493515, 0.5993322730064392, -0.767683207988739, 0.7402331233024597, -0.2201644778251648, -0.5824228525161743, 1.529891014099121, -1.4623613357543945, -0.7515428066253662, 0.10310597717761993, 0.9834180474281311, -0.3387315571308136, -1.1496011018753052, -1.735154628753662, 1.1390514373779297, -0.3799774646759033, 1.2958155870437622, 0.2266717255115509, 1.1658120155334473, 1.093063235282898, -0.3328430652618408, 0.19058817625045776, -1.5590122938156128, -1.0302727222442627, 0.3492527902126312, 1.7737587690353394, 0.08077282458543777, -1.8106932640075684, 0.37810707092285156, 0.3799072802066803, -0.12144225835800171, 1.6066912412643433, 1.1207343339920044, 0.5852230787277222, -0.2828647494316101, -0.1535123884677887, 1.0634993314743042, 2.582058906555176, 1.2938097715377808, 0.0006920917658135295, -0.1566513180732727, 0.2821453809738159, -0.3902377486228943, -0.1028832420706749, 0.3431715965270996, 0.9109461307525635, 0.9645214676856995, 0.9606838822364807, 0.3959742486476898, 0.5823321342468262, 0.8521769046783447, 0.002831466728821397, 0.4623275399208069, -0.41220101714134216, -0.6004076600074768, 0.2671867609024048, 0.28235286474227905, -0.6122490763664246, -0.5769213438034058, 0.9705400466918945, 0.43975716829299927, -0.9536064863204956, -1.1697747707366943, -1.273624300956726, 0.5340729355812073, 0.29400885105133057, 0.3746364414691925, 0.76839280128479, -1.0086584091186523, -0.16319076716899872, -0.050982680171728134, 0.7272839546203613, -0.4901919364929199, -2.519094944000244, -1.7765796184539795, 0.6971513628959656, 0.6331369876861572, 1.0207432508468628, -0.6718534231185913, -1.5400322675704956, -0.17703478038311005, -0.12954050302505493, -0.8854530453681946, 0.5080668926239014, -0.8265851736068726, 0.14316625893115997, -0.5796773433685303, 0.5759684443473816, 0.45668891072273254, 0.03101865015923977, 0.6850371956825256, 0.8484976887702942, -1.5941832065582275, 0.4804472029209137, -0.09664728492498398, -0.6760517954826355, 0.5107024908065796, 0.36070775985717773, -1.211694359779358, 1.2432711124420166, 0.05121500790119171, -0.7209429740905762, 0.4057571291923523, -1.15087890625, -0.2665804326534271, -0.2715752422809601, -2.0415420532226562, 1.4262683391571045, -0.8555765748023987, -0.3823908269405365, 0.2685082256793976, 0.33586785197257996, 0.47858983278274536, 0.3219672739505768, 0.6605144739151001, -2.025207757949829, 0.48072806000709534, 1.2820013761520386, 1.7841957807540894, 0.4206196963787079, -0.20473837852478027, -2.123033285140991, 0.5998789072036743, -1.4056150913238525, -0.1485314816236496, 0.06245125085115433, 2.1192989349365234, 0.26740819215774536, -2.333124876022339, -0.38197481632232666, 2.4803335666656494, -1.9878181219100952, -0.023828033357858658, -1.0849448442459106, -1.0113774538040161, 1.399765133857727, -0.6511701941490173, -0.7648597955703735, 1.7254639863967896, -0.8109320998191833, -0.6827202439308167, 0.4568864107131958, 0.5249390006065369, 0.796863853931427, 0.6624968647956848, 0.3593437075614929, -1.7743076086044312, 0.14644427597522736, 0.029026266187429428, -0.1560804545879364, 0.49242278933525085, 0.881763756275177, 0.6988674402236938, -0.3382127285003662, 1.6735903024673462, -0.07883495837450027, 0.6719686388969421, 1.4670872688293457, 1.0097543001174927, -0.4503244161605835, 0.6016126275062561, -1.0121724605560303, 0.7554646134376526, -0.6103189587593079, 0.8256421089172363, 1.0855913162231445, 0.7222868204116821, -0.43991395831108093, 0.010691714473068714, -0.09871869534254074, -0.17571713030338287, 0.6642283201217651, 0.1746530383825302, 0.8187850713729858, -0.04864594340324402, 0.46643000841140747, 0.16314134001731873, -3.0341951847076416, -0.7826547622680664, -1.5741552114486694, 1.0786672830581665, -3.1053991317749023, -2.3114216327667236, 0.10167818516492844, -0.48242971301078796, 0.8283038139343262, 1.9725546836853027, 1.898491621017456, 1.4647175073623657, -0.4732925593852997, 0.9785857200622559, 1.1282438039779663, -0.27670109272003174, -1.3581347465515137, 0.8714035749435425, -1.3855979442596436, -1.1298567056655884, -0.1577093005180359, -0.011801082640886307, 0.4524500370025635, 1.3065146207809448, -1.4484889507293701, -0.8428210020065308, -0.3752090334892273, -0.07999780774116516, -0.13544271886348724, 0.3221725523471832, 0.18595726788043976, -0.20430123805999756, 0.5105379223823547, -0.6407559514045715, 0.7662557363510132, 0.029251674190163612, -0.21481963992118835, 0.12197287380695343, 0.44199705123901367, 1.0049831867218018, -1.0390387773513794, -1.5692791938781738, -0.5458787679672241, -0.3449781835079193, 0.28726470470428467, -0.12536214292049408, 0.659303605556488, 0.3977723717689514, 1.7850507497787476, -1.0823066234588623, -0.4134245812892914, -0.11510488390922546, 1.3888558149337769, -0.49895089864730835, 0.2561301589012146, -0.989176332950592, 2.1318538188934326, 0.9798007607460022, 1.597185492515564, 1.2398014068603516, 0.9922295212745667, -0.15445467829704285, -1.184261441230774, 1.8842092752456665, -0.5354131460189819, 2.1607065200805664, 0.0613977275788784, -0.2534020245075226, -1.0278886556625366, 0.11783839017152786, -0.6345401406288147, -0.019575033336877823, -0.5076760649681091, -0.3950890302658081, -1.090846300125122, -0.6690207719802856, -0.6512265205383301, -0.7627862691879272, -0.8867823481559753, 0.6609954237937927, 0.7786296010017395, 0.6812378764152527, -1.5320701599121094, 0.657370924949646, -0.019546356052160263, -0.7755079865455627, -1.9909590482711792, 0.5269811153411865, 0.2491019368171692, 0.6436522603034973, -2.1739368438720703, -0.5154236555099487, 0.5045256614685059, -0.2922569513320923, 0.26669546961784363, 0.3355296552181244, -0.5809900760650635, -0.47861340641975403, -0.8482822775840759, 0.46048152446746826, -0.8993757367134094, 0.37596526741981506, -0.04678324609994888, 0.8213204145431519, -0.6340076923370361, 0.38698068261146545, -0.7341083288192749, 0.46816286444664, -0.05101959407329559, 0.7556187510490417, 0.9278303384780884, -0.581951379776001, 0.8693869709968567, 1.8709967136383057, -0.7426553964614868, 0.5891475677490234, -1.946598768234253, -0.5541965961456299, -1.03793203830719, 0.46292221546173096, -0.9895591139793396, -1.778559923171997, 1.8233145475387573, 0.01842719316482544, 0.755784809589386, -0.5150158405303955, -0.8386889100074768, 1.1162450313568115, -0.22844180464744568, -2.3204596042633057, -1.8639891147613525, -0.832160234451294, 2.0510990619659424, -0.36449846625328064, -0.4320855438709259, -0.5562828779220581, 0.3824208974838257, 0.20495480298995972, 0.4696206748485565, -0.7940353155136108, 0.4865427017211914, -0.03672230616211891, 0.0011339165503159165, 0.11308751255273819, 0.6284050345420837, 1.0811470746994019, 3.106194257736206, -0.7221593260765076, 0.5851312875747681, -0.8337441682815552, 0.40305081009864807, 0.5823430418968201, -0.25612011551856995, 1.0871458053588867, 0.7555540204048157, 1.191401481628418, 0.246347114443779, -1.2081750631332397, 1.3368315696716309, 2.045116662979126, 0.5406984686851501, -0.5137956142425537, 1.4753711223602295, 0.16915106773376465, -2.001993179321289, -0.8504854440689087, -1.477645754814148, 0.7709214091300964, -0.3776281476020813, -0.9278085827827454, -0.5862045288085938, 0.8407983183860779, -2.070185422897339, -1.4258004426956177, 0.05876363068819046, -0.7138370275497437, -0.6692479848861694, -3.23266339302063, 1.2751437425613403, 1.313689947128296, -0.14736655354499817, -1.092164397239685, -0.40773507952690125, 0.7110247015953064, 0.21855589747428894, 0.2618940770626068, -0.824114203453064, -1.1030632257461548, -0.8496323823928833, 0.9177854657173157, -0.06744076311588287, -0.9457482695579529, -0.07298087328672409, 2.250437021255493, -0.3830839991569519, 0.26568785309791565, 0.314399778842926, -1.6836985349655151, 1.5302823781967163, -0.5430036783218384, 0.8645245432853699, -0.24696926772594452, -0.538499116897583, -0.18187430500984192, -0.47026804089546204, 0.4145716428756714, -0.10907536000013351, 0.017694171518087387, 0.6455444693565369, -1.3326612710952759, 0.2799971401691437, -0.05166886746883392, 0.1925470530986786, -0.5070628523826599, 1.2439085245132446, -0.3198689818382263, 1.4265574216842651, 2.1032049655914307, -0.8895328044891357, -1.560684323310852, 0.2749270796775818, 0.08201684057712555, 1.034415364265442, -0.04163585603237152, -1.142830491065979, -2.666023015975952, -1.8478947877883911, -0.5130559802055359, 0.5057159662246704, -0.5393843650817871, 0.10152052342891693, 1.1171523332595825, -0.027597609907388687, -1.7054922580718994, 0.6883901357650757, -1.1993910074234009, -0.48318013548851013, -0.20370958745479584, -2.0067062377929688, 1.1508128643035889, 0.02006307803094387, -0.022909972816705704, -0.054604750126600266, 0.10677022486925125, -0.3024340569972992, 0.391916424036026, 0.35090720653533936, -0.1861717700958252, -0.6689027547836304, -0.056791752576828, 0.17070181667804718, 1.1865975856781006, 0.8894209265708923, -0.057849857956171036, 0.3076932430267334, 1.5673408508300781, 2.059934139251709, 0.4694601893424988, 1.3750547170639038, -1.676668643951416, 0.815963864326477, 0.5332709550857544, -0.8106884956359863, 0.8595619797706604, 0.94608074426651, -0.9637805223464966, 0.7695331573486328, -1.6680965423583984, 0.4005042612552643, -0.5745700597763062, 0.06370384991168976, -1.0451303720474243, 0.8261092305183411, -1.3265246152877808, -0.7106504440307617, 1.3682414293289185, -0.17741777002811432, -0.6731745600700378, 0.5655530691146851, 1.145573377609253, 0.8167053461074829, 1.2777875661849976, 0.6300109028816223, 0.012215918861329556, 1.5847359895706177, -0.8448310494422913, 0.2703576385974884, 0.9059585928916931, -1.217613697052002, 0.21721459925174713, -0.27880918979644775, 2.0165457725524902, -0.19740799069404602, -0.7174474596977234, 0.1216939240694046, -1.252913475036621, -0.03660058230161667, -0.2300729900598526, -0.3131382465362549, 0.1315440684556961, -0.5273580551147461, -0.7652167677879333, 1.0225569009780884, 0.2524375915527344, -0.5516949892044067, -0.4010230004787445, 0.6710977554321289, 1.238073468208313, -0.4524422585964203, 1.2536308765411377, -0.07121596485376358, 0.4462023079395294, 1.7835692167282104, -2.1916708946228027, 1.4669897556304932, -0.12139041721820831, -1.2502063512802124, -0.9472121000289917, -0.5887227058410645, 0.24166424572467804, -0.2978449761867523, 0.6110292673110962, -1.449890375137329, -0.30660784244537354, -0.33194735646247864, -0.3817351758480072, -0.006633498705923557, 1.2703958749771118, 0.6870119571685791, -1.9188092947006226, -0.9483547210693359, -2.076754331588745, 0.49347180128097534, -1.1920150518417358, 0.10519513487815857, 1.4495245218276978, 0.6879311800003052, 0.44657978415489197, -1.0878880023956299, -1.0901182889938354, 1.5230004787445068, 1.2055506706237793, -0.3605167865753174, -1.1938800811767578, 1.1514253616333008, -1.6124169826507568, -1.1456236839294434, -0.09365854412317276, -1.2227188348770142, -0.42793968319892883, 0.9690951704978943, 0.08344284445047379, -2.0761001110076904, -0.06351660192012787, 0.1890309453010559, 0.9142037630081177, -0.23626695573329926, 0.5690272450447083, -1.092847228050232, 1.0820142030715942, -1.0537596940994263, -0.48558366298675537, -0.7836534976959229, -1.3085628747940063, -0.008938047103583813, 0.23319421708583832, -1.481507658958435, 1.1344540119171143, 0.27132976055145264, -0.258184015750885, -0.08028441667556763, 0.26650339365005493, 0.7903843522071838, 0.19948536157608032, -0.8220967650413513, 0.7680237293243408, -1.2576242685317993, 1.2859381437301636, 0.8193349838256836, 1.5255444049835205, -0.9045916199684143, 0.6385942697525024, -0.3490128517150879, 1.2512539625167847, -0.2848275601863861, -1.4262259006500244, -0.003921946045011282, 0.8752949833869934, 0.3612726032733917, 1.0681263208389282, -0.4606974422931671, 0.6202778816223145, 1.3075751066207886, -0.7064317464828491, -1.2709624767303467, 1.179627776145935, -0.21000030636787415, -0.9630364775657654, 0.09489254653453827, 0.6126640439033508, 1.7371083498001099, -2.064958333969116, 0.22948676347732544, -0.9537087082862854, -0.7568747401237488, -1.5416488647460938, 0.6083720922470093, -1.2116361856460571, -1.4510345458984375, -0.17832332849502563, 0.7785628437995911, -0.010067605413496494, -0.17981496453285217, -2.406858205795288, 1.102604627609253, 1.3410170078277588, 0.8751574158668518, -1.4792824983596802, 0.7911549806594849, -0.9235938787460327, 0.4539540410041809, -0.21018587052822113, 0.6917294859886169, 1.7896019220352173, 0.6504544615745544, 0.5821523666381836, 0.07773791253566742, 2.1764888763427734, 0.6107674837112427, -1.051172137260437, -1.087589144706726, -0.48041802644729614, 0.09824869781732559, 0.503585934638977, -1.5978684425354004, -0.5165067315101624, -0.6700710654258728, 0.01812799647450447, 1.576499342918396, 0.3118381202220917, -1.4699797630310059, -2.108818292617798, -1.1161754131317139, -0.0017567857867106795, 1.226770281791687, 0.24026000499725342, -0.9056702852249146, -0.25047504901885986, -0.1644192636013031, 0.7544682025909424, 0.6318981647491455, -0.5243092775344849, -0.929365336894989, 1.4983874559402466, 0.12625615298748016, 1.2375327348709106, -0.47127893567085266, -0.22702975571155548, 0.5664570927619934, -0.7285947799682617, -0.18768435716629028, -0.4569646120071411, -1.922642469406128, 0.6375253200531006, -0.15593978762626648, -0.32835036516189575, -0.5434858202934265, 0.8996596336364746, -1.3654382228851318, 1.043663501739502, 1.5092246532440186, -0.6014529466629028, -0.003016777103766799, 0.06848318874835968, 1.2573212385177612, -0.6612300872802734, -0.09165160357952118, 0.23330014944076538, 1.2300338745117188, -2.123140335083008, 2.8981921672821045, 0.2531883418560028, -0.011497419327497482, 0.8780294060707092, 0.2868286967277527, -0.8416383266448975, -1.5363953113555908, 0.10057452321052551, -2.062086343765259, 0.5049983263015747, 0.3517681360244751, 0.4859866797924042, -0.3533841371536255, -0.21752698719501495, 0.172127828001976, 0.7837414145469666, -0.028717435896396637, 0.23075102269649506, 0.31488412618637085, 1.3775631189346313, 0.6053457260131836, -0.11705002933740616, 0.6184561252593994, -0.04999496415257454, 2.0398871898651123, 0.6392151117324829, 0.859715461730957, 1.2839146852493286, 1.5174013376235962, 0.8400999903678894, -2.4511284828186035, 1.3625134229660034, 0.9742932319641113, 0.37232449650764465, 1.9662830829620361, 1.2381693124771118, 1.6971228122711182, -1.0810565948486328, 0.9211062788963318, 1.7737221717834473, -0.017802445217967033, 0.44645076990127563, -0.46781450510025024, 1.1028701066970825, -0.1962316781282425, 1.2244879007339478, 0.7320669293403625, 0.3325871527194977, -0.4275805652141571, 0.4242362976074219, -0.8232086896896362, -1.4599287509918213, 0.3967377841472626, 0.5474357008934021, -0.4635046422481537, 2.6592297554016113, 0.1467914879322052, -0.32093411684036255, 0.1548246294260025, -0.14425119757652283, 0.912053644657135, -0.047841183841228485, -0.2880943715572357, -0.1370009332895279, -0.824137270450592, -0.6462000012397766, 0.044126685708761215, -1.2730220556259155, -0.9892352223396301, -1.4127880334854126, 0.26546454429626465, -0.6903793215751648, -0.028770066797733307, 0.2957031726837158, -0.4761356711387634, -0.02179492451250553, -0.08047459274530411, 0.17962545156478882, -0.06374705582857132, 1.8094911575317383, -1.056842565536499, -1.4005544185638428, 0.3998931050300598, -0.6037532091140747, 0.6160948872566223, 0.12261836230754852, -1.632435917854309, 0.9380447268486023, 0.9188660979270935, -0.16775724291801453, 0.4823663830757141, 0.17315758764743805, -1.8722230195999146, 0.3015235662460327, 0.5859805345535278, 0.7302855849266052, -1.9075483083724976, -0.5438328385353088, 0.7394016981124878, 0.18134750425815582, 0.5613102912902832, -0.6380853056907654, 0.49120303988456726, -2.818344831466675, 0.7727507948875427, 1.3137644529342651, 0.1999780833721161, 0.7043525576591492, 1.0794016122817993, -2.0219011306762695, -1.2963603734970093, 2.5629711151123047, -0.7539967894554138, 1.4648151397705078, -0.1214665025472641, 1.4895199537277222, -0.597625195980072, -0.8430339097976685, -1.227134346961975, 0.2681806683540344, 0.6853399276733398, -0.6106294989585876, 0.27520957589149475, 2.2947075366973877, -1.4813066720962524, -3.153287172317505, -0.7270727157592773, -0.7180801033973694, -0.3827316164970398, 0.22188150882720947, 1.3701541423797607, -0.736823320388794, -0.9275065660476685, -1.5109981298446655, -0.7649011015892029, -2.175230026245117, -0.283834844827652, 0.559677004814148, 0.12732505798339844, 0.3398948907852173, -0.509394645690918, 0.22024954855442047, -0.08466852456331253, 1.8515549898147583, 0.1945425122976303, -0.992949366569519, 0.4090757966041565, -0.5560218691825867, -1.4595247507095337, -0.7634525299072266, -0.41115495562553406, -1.9182279109954834, -1.1482480764389038, -1.3158366680145264, 0.43922558426856995, 0.01921295002102852, 0.4800192713737488, 0.0619005411863327, 0.6112182140350342, -0.4073324501514435, 0.035849180072546005, -1.397889256477356, 0.04621710255742073, 1.0369781255722046, 1.3426657915115356, 0.2718609869480133, 0.2917343080043793, 0.564271092414856, 0.22985276579856873, -0.9548673629760742, 1.2717154026031494, 0.6377724409103394, -1.0903080701828003, -2.015373945236206, 1.6434656381607056, -0.8750675320625305, 0.5976875424385071, -0.6823523044586182, 0.5966314673423767, 0.07165466248989105, 0.37642422318458557, -0.4547002613544464, -1.0475116968154907, -1.9420005083084106, -0.6273888945579529, -1.5387651920318604, -1.8820509910583496, 0.0066075874492526054, -0.35646316409111023, -1.761014461517334, -2.0074005126953125, -0.8958686590194702, 0.26783978939056396, -0.6487219929695129, -0.33199477195739746, -0.2884911298751831, 1.352691650390625, 1.0588855743408203, -2.413876533508301, -1.3618769645690918, 1.01211416721344, 0.8836167454719543, -0.7270538210868835, -0.12657761573791504, 1.0474178791046143, -0.5996870994567871, -0.02566218562424183, 0.19084377586841583, 0.48617202043533325, -0.33120980858802795, -0.2508818805217743, -0.14367444813251495, 0.019070876762270927, 0.1709260642528534, 0.24514053761959076, -0.9942766427993774, -1.036838173866272, 0.4731255769729614, 1.2027555704116821, 0.4969688951969147, -0.5618849992752075, -0.7239677906036377, 1.256286382675171, -0.08336074650287628, -0.2534679174423218, 1.1799389123916626, 1.0406572818756104, 0.10400457680225372, 1.9511646032333374, 2.455916404724121, -1.5910507440567017, 0.03464808687567711, -1.7446517944335938, -0.5836526155471802, 0.05453316122293472, -0.7154651880264282, -0.7782609462738037, -1.341081976890564, 1.4473971128463745, -0.48940494656562805, 1.318203330039978, 0.8029482364654541, -0.4014925956726074, 0.4646054208278656, 0.72153240442276, -0.6148134469985962, 0.45237720012664795, 0.9402610659599304, 0.9007592797279358, -1.2027395963668823, -0.7177324891090393, -1.165602445602417, -0.2622072100639343, 1.0375362634658813, -0.9412940740585327, 2.4332501888275146, 0.12776486575603485, 1.31487238407135, -0.4561152160167694, 0.7937257289886475, 0.02693602256476879, -0.6955545544624329, 0.5952140092849731, 0.703850507736206, -0.6164886951446533, 0.3971013128757477, -0.37934136390686035, -0.10072661191225052, 0.361240416765213, 0.2622562348842621, 1.1371711492538452, 0.10588634759187698, -0.34068453311920166, -0.5505287051200867, -1.2273914813995361, 0.9064257740974426, -1.5376571416854858, 1.000065565109253, 0.029917342588305473, -0.3428671956062317, 0.25715550780296326, 0.3236677944660187, 1.696763038635254, -1.2569613456726074, -1.965039610862732, -0.9681841135025024, -0.5626481771469116, 0.3079955577850342, 0.28244343400001526, -2.110501289367676, -1.8976582288742065, -0.4554346203804016, 1.4867054224014282, 1.4759997129440308, 1.1003443002700806, 0.41542190313339233, -0.8646711111068726, -0.5448372960090637, -1.0627115964889526, 0.999942421913147, -1.4391812086105347, 0.6284497976303101, -0.786541759967804, 0.6824636459350586, -1.5122250318527222, 0.17955659329891205, 0.05029914528131485, -0.8225695490837097, 0.11433826386928558, -1.8867725133895874, 1.3509628772735596, 0.4007525146007538, -1.620856523513794, -1.2268303632736206, -2.0350053310394287, -0.19846579432487488, -1.3247779607772827, 0.22495019435882568, 0.7394112348556519, 1.0457229614257812, 0.3016778230667114, 0.0797848254442215, 0.7215770483016968, 1.0792906284332275, -1.2134795188903809, -0.294053852558136, 0.7383449077606201, -0.3895726203918457, 0.6629940271377563, -1.0598052740097046, -1.8081912994384766, -0.011364046484231949, 0.03346918523311615, 0.2672904133796692, -0.08905652910470963, -0.2122911959886551, -0.5385032296180725, -2.3063228130340576, -1.2631292343139648, 0.2962721288204193, -0.38676881790161133, 0.20232051610946655, 1.6654229164123535, 0.4816271662712097, 1.178171992301941, -1.8999159336090088, -1.4036850929260254, -0.6458301544189453, -0.1392209231853485, 0.6739944219589233], [-0.4231972396373749, -0.8451719284057617, -1.798951268196106, 1.7668994665145874, 1.9325562715530396, -0.30796337127685547, -0.7952941060066223, 0.10275238007307053, 0.1975402981042862, -2.396174669265747, 0.5993216037750244, -0.4031710624694824, -1.3887743949890137, -0.7074457406997681, -0.27043136954307556, 0.24318218231201172, -0.6629738807678223, -0.7807794809341431, -1.0505083799362183, -0.6806055903434753, -0.48012351989746094, 0.45421528816223145, 0.5215489864349365, -0.8373070955276489, 2.500281572341919, -1.3193325996398926, -0.5814968943595886, -1.4233753681182861, -0.48371410369873047, -0.05829928070306778, -0.5756458640098572, 1.8880163431167603, 1.590811848640442, 0.4986313283443451, 0.7891910672187805, -1.2720097303390503, 0.2460874617099762, -0.644443690776825, -0.17906031012535095, 1.0836104154586792, 1.6421445608139038, -0.09442248195409775, -0.4717094898223877, 1.2835261821746826, 0.923021674156189, 0.10052884370088577, 0.25649821758270264, 0.8043531775474548, -0.11093388497829437, -2.014723062515259, 1.0349754095077515, -0.17573313415050507, -1.2772538661956787, 0.20604412257671356, -1.3232396841049194, -0.9795724153518677, -0.7727273106575012, 0.4296964406967163, -1.2140923738479614, -0.19111181795597076, -0.2645736336708069, -0.34650519490242004, 1.265047550201416, 1.0777359008789062, -0.316732257604599, -0.08858352154493332, -1.423360824584961, -0.9641006588935852, 1.0780285596847534, 0.7765868306159973, 1.373744249343872, -1.236583948135376, 0.5387071371078491, 0.4210614860057831, -0.11767076700925827, 1.0091346502304077, -0.9373661875724792, -1.82008957862854, -0.6085218191146851, -2.608393907546997, 0.9736571907997131, -0.7450806498527527, 0.49545061588287354, 0.540120005607605, 0.5459954738616943, 0.39249372482299805, 0.8199993968009949, 1.313600778579712, -0.5160762667655945, -0.25484803318977356, 1.3927351236343384, -1.0210719108581543, 0.7339136004447937, -0.4610755443572998, 0.8513447046279907, -0.9296367764472961, -0.6242226958274841, 0.006505716126412153, 1.0496411323547363, 0.9615683555603027, 1.0549196004867554, 0.7520782947540283, 2.269831895828247, -0.7463850378990173, 0.48921018838882446, -0.4741903245449066, 0.09406107664108276, 0.9706975221633911, -0.1773408055305481, -0.5143135786056519, 1.1512110233306885, -0.41428494453430176, -0.17436085641384125, 1.444202184677124, -0.42457351088523865, 0.4281214773654938, 0.4923812448978424, -0.6529189348220825, 0.4512833058834076, 0.3902660608291626, 0.7670902609825134, -1.0736613273620605, -1.32988703250885, 1.9165107011795044, -0.3662164509296417, -0.33724820613861084, 1.1123220920562744, 1.2624109983444214, 1.4209476709365845, -1.0460829734802246, -0.01840309239923954, -0.20248515903949738, -2.572155237197876, 0.5480031967163086, 0.023798972368240356, 0.16545648872852325, 1.574371099472046, 0.04466503858566284, -0.36025190353393555, 0.6477200984954834, 1.65647554397583, 0.6368241906166077, 0.012571346014738083, 0.849293053150177, -0.032722994685173035, 1.3552945852279663, -0.3369980454444885, -0.92604660987854, -0.9845722913742065, 0.17930100858211517, 0.24401205778121948, 0.7549331784248352, -0.4262634515762329, -0.32303524017333984, 0.47309696674346924, -0.6832359433174133, 1.0475245714187622, -1.232140064239502, 1.3428478240966797, -1.088377833366394, 0.7666777968406677, -0.3064865469932556, 1.094710111618042, -1.7508010864257812, 0.1548018902540207, -1.3246595859527588, 0.3969985246658325, 0.5249686241149902, -0.40984246134757996, 0.1213412880897522, 1.4591827392578125, 1.2304049730300903, 0.18242104351520538, -0.615618884563446, 0.034242600202560425, -1.3708131313323975, 0.15799568593502045, -0.10674452781677246, -0.889365017414093, -0.5432149171829224, -1.799342393875122, 0.025750234723091125, 0.7794306874275208, 1.3140658140182495, 1.1778464317321777, 1.1114646196365356, 1.9950158596038818, -0.7140291929244995, -1.516746163368225, 0.1308673471212387, 2.3408262729644775, 0.2797531187534332, -1.4432586431503296, 2.088204860687256, -0.9460572004318237, 0.23004284501075745, -0.5076348185539246, -0.6171112656593323, -0.7140849828720093, -0.6044849753379822, -0.36677810549736023, -0.7722639441490173, -0.38075023889541626, -1.50031578540802, 0.0893142819404602, 0.47776493430137634, 0.7467474341392517, -0.904505729675293, 0.1561003029346466, -0.5387976765632629, -1.1313303709030151, 1.0317847728729248, -0.005295925308018923, -0.8617796301841736, 0.8351742625236511, -0.13173387944698334, -0.647347629070282, 0.5074419975280762, -0.42552345991134644, 2.641005754470825, 1.4036264419555664, -0.9756343960762024, 1.8564599752426147, 0.19559814035892487, 1.3802268505096436, 0.7069304585456848, 0.8009519577026367, -0.025702690705657005, -0.4183569848537445, 1.1805474758148193, 1.2066318988800049, 0.7246183156967163, -1.8154147863388062, 0.802736222743988, -0.45933833718299866, -0.4379200339317322, 0.6698578596115112, -0.7487413287162781, 0.44767650961875916, -0.2501629889011383, 0.9026058912277222, 0.6350400447845459, -0.1356116533279419, 0.1343556046485901, -0.19878803193569183, -0.2778185307979584, -0.313739538192749, -1.1603734493255615, -0.6662508249282837, -0.6405414938926697, 0.8862713575363159, 1.0209736824035645, 0.41959840059280396, 1.5328305959701538, 1.7791039943695068, -1.262291669845581, -1.394864797592163, 1.2827342748641968, 0.4717548191547394, 0.216196671128273, -1.1967958211898804, -1.6336097717285156, 0.13692066073417664, 0.8891228437423706, 1.031076431274414, -2.445119857788086, 0.016434308141469955, 1.4849276542663574, 0.5205460786819458, 0.18624667823314667, -0.04238901287317276, 1.8886291980743408, -0.9213343262672424, 0.048545025289058685, -2.444514513015747, 0.7886998057365417, 1.687232255935669, -1.328233003616333, 2.8253109455108643, 1.5009829998016357, 1.0184903144836426, -1.0539199113845825, 0.19023239612579346, -1.3477927446365356, -1.277677059173584, 0.6856370568275452, -1.4389454126358032, 0.5546298623085022, -1.1750502586364746, -0.3239956498146057, 0.35147690773010254, 0.3357784152030945, 0.9890359044075012, -0.3143429458141327, -2.020343065261841, 1.1007838249206543, -0.7271782159805298, 1.1149771213531494, 0.20137330889701843, -1.3087424039840698, 1.674284815788269, 0.596234142780304, 0.40122631192207336, 0.32482755184173584, -0.12806783616542816, 1.1932991743087769, 1.2529621124267578, 0.3570505678653717, 0.35236474871635437, 1.3877010345458984, 0.45864617824554443, -1.2812007665634155, 1.138203501701355, 0.1600351482629776, -1.1499136686325073, -0.10018233954906464, 1.4624927043914795, -0.06569166481494904, 0.5437507033348083, 1.1257144212722778, -0.6199864745140076, 1.5508497953414917, -1.3506022691726685, 2.3218133449554443, 2.223583936691284, -0.40225037932395935, 1.4992060661315918, 0.7678605914115906, -0.6844301223754883, -0.03998924791812897, -0.49991244077682495, -1.2879793643951416, -0.355622798204422, -0.15378950536251068, 0.3652976453304291, -1.1326584815979004, -1.7628253698349, -1.7793623208999634, 0.9405513405799866, 0.6594284772872925, -0.6824328899383545, 0.09121313691139221, -1.6465575695037842, -0.41955992579460144, -1.4198747873306274, 0.49070999026298523, -0.9936675429344177, 2.7960267066955566, 1.771327257156372, 1.0386601686477661, -0.9867429733276367, 0.380878210067749, 0.10643833875656128, -0.2398747056722641, -0.7960993647575378, -0.6055089235305786, 0.14819586277008057, -0.3391134440898895, -0.4423876702785492, 2.5302770137786865, -0.503223180770874, 2.6900200843811035, 0.05719031020998955, 1.424239993095398, -1.6194441318511963, 0.9415491819381714, -0.6125470995903015, -1.678033709526062, -2.080432653427124, -0.4414530396461487, -0.5324617028236389, -0.2627248764038086, 0.03777230903506279, -1.5442744493484497, -0.6628807783126831, -0.7971612811088562, -0.10682497918605804, -0.42656728625297546, -0.29020994901657104, -1.6730679273605347, -0.11998306214809418, 0.05372792109847069, 0.8758276104927063, 0.09067612141370773, -0.13527847826480865, -0.21163906157016754, -1.3129760026931763, 1.1091262102127075, -0.8052045702934265, -0.2282848358154297, -0.6488645672798157, -0.7589545845985413, 0.4209751784801483, 0.43519264459609985, 0.5424938201904297, -0.0026368650142103434, 2.0728609561920166, -1.0564801692962646, -0.36902478337287903, -2.095081329345703, -0.5826160907745361, 0.09241237491369247, 0.8062483072280884, -1.2162715196609497, 0.5975514650344849, -0.4690823554992676, 0.16367031633853912, 1.706310510635376, 0.3426067531108856, 0.23037996888160706, 0.43206486105918884, -1.9962910413742065, -0.02937394194304943, -2.163139820098877, -1.9452850818634033, -0.01938006281852722, -0.9897955656051636, -0.1514565646648407, -1.3090803623199463, -0.4195742905139923, -1.0080840587615967, -0.8471534848213196, -0.8764227032661438, 1.012824296951294, -0.3939533829689026, -0.4975987374782562, 0.5681438446044922, -0.1409836858510971, -0.429787278175354, 1.1407787799835205, -1.2773895263671875, -0.6394920349121094, -0.1701892465353012, 0.6461948156356812, -0.5997126698493958, 1.436858892440796, -1.6856435537338257, 1.2428289651870728, -2.2693042755126953, -0.5178142786026001, 0.6564819812774658, 1.8033654689788818, 0.9517102241516113, 0.2542392909526825, -0.4915851950645447, 0.6639434099197388, 0.6103638410568237, -1.005098819732666, 0.32369083166122437, -0.06981565803289413, -0.2876339852809906, 0.08273325115442276, -0.7719850540161133, -1.129403829574585, -0.2755882441997528, -0.8443731665611267, 0.18486115336418152, 1.4256056547164917, -1.4242242574691772, 1.632582664489746, -1.8427621126174927, -0.00043623513192869723, 1.882226824760437, 0.41377151012420654, -1.0689313411712646, 0.6171501874923706, -1.5916513204574585, -0.6513630151748657, -0.6022994518280029, 1.1841603517532349, -0.5615508556365967, 0.1831786334514618, -0.10508298873901367, -0.9462371468544006, -0.49144694209098816, -1.269073486328125, -0.6087780594825745, 0.5355337858200073, -0.1328563094139099, -0.030468637123703957, -0.1592477262020111, 0.8579272031784058, -1.1709041595458984, -1.2621574401855469, -1.4886798858642578, -0.8367930054664612, -0.0008832791354507208, 0.8850345611572266, 1.0242297649383545, 0.19072112441062927, 1.5647300481796265, 1.5035115480422974, -0.9686440229415894, -0.01987660489976406, 1.1853891611099243, -1.6647311449050903, -0.83188796043396, 2.004852056503296, 0.153233140707016, -1.1804529428482056, 0.05576450750231743, 0.9254804849624634, -0.2769344747066498, 0.23385989665985107, 0.45913392305374146, -0.09568922966718674, -1.0171449184417725, -0.7888066172599792, -0.7923603653907776, 0.43317216634750366, 0.5043734312057495, 0.7771950364112854, -1.0408860445022583, 0.024734899401664734, -0.22171860933303833, -0.26958104968070984, -1.134666085243225, 1.3603166341781616, -0.07515248656272888, 0.3691617548465729, 0.4189065992832184, -0.759553074836731, -0.44495299458503723, 0.5519749522209167, -0.6579742431640625, -2.202394485473633, -0.1667596846818924, 0.23136767745018005, -1.3633660078048706, -1.1001046895980835, -0.5727656483650208, -1.0067412853240967, -0.9122376441955566, 1.3742828369140625, -0.06845197826623917, -2.949561834335327, -2.9801337718963623, -0.7486423850059509, -1.2041914463043213, -1.1813061237335205, -0.34160616993904114, 0.18180809915065765, -0.46095436811447144, 1.1944553852081299, 0.5335429310798645, 0.2949480712413788, 0.47775349020957947, -1.0788273811340332, -0.2710901200771332, -1.2259597778320312, 0.5847880840301514, 1.336863398551941, 0.488931804895401, -0.4274919927120209, -2.05424427986145, -0.17685307562351227, -0.9055496454238892, -1.210019826889038, -1.7327812910079956, -0.009804067201912403, -0.6469402313232422, -1.001761555671692, -0.24718861281871796, -0.0551321879029274, 2.0498945713043213, 0.2533312141895294, 1.654347538948059, -0.9498160481452942, -0.6259109973907471, -0.6510094404220581, -0.8743277788162231, -0.17114000022411346, -1.0995588302612305, -0.7184056043624878, 0.5636264681816101, -0.1497262418270111, 1.7053836584091187, 0.5142254829406738, -0.02902498096227646, -2.1953842639923096, 0.5083463788032532, -1.3278930187225342, 1.9044365882873535, -1.5401312112808228, 0.5489124059677124, -2.1164159774780273, -1.1192419528961182, 0.27762478590011597, -0.4436476230621338, 0.13325294852256775, -0.6522310972213745, -1.0923742055892944, 1.3218556642532349, -0.638600766658783, -1.6631262302398682, -0.8006337881088257, -1.3204119205474854, 0.04733766242861748, -1.6633185148239136, 0.29857391119003296, 0.21960483491420746, 0.1245935782790184, 0.4834901690483093, -0.608812689781189, -0.9191555380821228, 0.5863468050956726, 0.21782009303569794, 1.2184233665466309, -0.05973730608820915, 1.2302583456039429, 0.43921223282814026, 0.03633980453014374, -0.7602755427360535, 1.1391857862472534, 0.09867103397846222, 0.2969590425491333, -1.2277415990829468, 0.11421696096658707, -1.0714765787124634, -0.20789876580238342, -1.5168876647949219, 0.33008715510368347, -0.5406826138496399, -1.1695798635482788, -0.5619919896125793, -0.028302615508437157, 0.5122376680374146, -0.47060084342956543, -2.3305091857910156, -1.344462513923645, 0.024221383035182953, -0.5931407809257507, -0.05292307585477829, 0.4019910395145416, 0.4950907826423645, 0.5346019864082336, 0.014856373891234398, -1.512755274772644, -0.4699782133102417, -2.032470226287842, -0.5735830068588257, -0.10702124983072281, 0.8042521476745605, 0.022587850689888, -0.5726760625839233, -0.1327516734600067, -0.3197803795337677, 0.030517954379320145, 1.4712671041488647, 2.7753169536590576, -1.181954264640808, -0.04679739102721214, -1.3028888702392578, -0.5445350408554077, -0.9951319098472595, 0.22334107756614685, -0.7637907862663269, 1.3194228410720825, 0.5898045301437378, -0.10375995188951492, -2.4967238903045654, 0.32840412855148315, 0.37961217761039734, 0.5118257999420166, -0.20868542790412903, -0.0013091436121612787, 1.3627530336380005, 0.61947101354599, 0.3036889135837555, 2.20758318901062, -0.8614111542701721, 0.28856778144836426, -0.47761356830596924, -1.1414239406585693, -0.7111329436302185, -1.287339687347412, 1.7695714235305786, -0.4006493389606476, 0.719455361366272, 2.3309648036956787, -0.885779619216919, 0.2514404058456421, -0.38320934772491455, -1.0097397565841675, 1.204925298690796, -2.393852949142456, -2.108168840408325, 0.2870226502418518, -0.14940933883190155, -0.12113240361213684, 0.9372885227203369, -0.48293933272361755, 0.2805744409561157, 0.038429729640483856, -0.522631824016571, -0.08548631519079208, -1.0424376726150513, -0.7888826131820679, 0.4422673285007477, -0.14444561302661896, 0.33824270963668823, 1.0848288536071777, -0.1669880896806717, 0.07099997252225876, 0.7793895602226257, -2.1226980686187744, 0.22028151154518127, -1.035354733467102, -0.5238866209983826, 0.15067510306835175, -1.6001769304275513, -1.0364872217178345, 0.2465212643146515, -1.4622148275375366, -0.049519844353199005, 1.2321453094482422, -1.2867168188095093, -0.04246709868311882, -0.15955843031406403, 0.2284119427204132, 1.6574220657348633, -1.5667372941970825, -0.6523547768592834, -0.9857499003410339, 0.11027213931083679, 0.11339154839515686, -0.13157986104488373, -0.5169065594673157, -0.41759827733039856, -1.6767991781234741, 2.4277474880218506, -0.1362985521554947, 1.0464602708816528, -0.7555104494094849, 0.0008287184173241258, -0.5652737021446228, -1.9668405055999756, 2.009951114654541, -1.3794856071472168, 1.515400767326355, 0.9356083273887634, -0.5084841251373291, 2.712740898132324, 1.5301997661590576, 0.02527141012251377, -0.2818373441696167, 0.2074713259935379, 0.09517795592546463, -0.02909507229924202, -0.41508013010025024, -0.21630500257015228, -1.3927961587905884, -0.002715296810492873, 1.0879276990890503, 0.879854142665863, 1.0157620906829834, 0.12035296857357025, 0.030668262392282486, -1.10640549659729, -0.4904847741127014, 1.563673496246338, 0.1418450027704239, 0.15481054782867432, -0.5358278751373291, 1.083244800567627, -0.19854401051998138, 0.5417389273643494, -0.033614736050367355, 0.015809636563062668, -0.7302667498588562, -0.28875982761383057, -0.4831245243549347, -1.0338749885559082, -0.6951255798339844, -0.005936791189014912, 2.1329619884490967, -1.3359031677246094, 1.4456908702850342, -0.7979296445846558, 1.0000189542770386, 0.10911253839731216, 0.07641523331403732, 0.0470019169151783, -1.843418002128601, 0.47780609130859375, -2.5617833137512207, -0.3429834544658661, -0.567546010017395, -1.0225151777267456, 1.2101435661315918, 1.7752184867858887, 0.44719046354293823, 0.6434444189071655, 0.7947997450828552, 1.5751609802246094, 1.4703006744384766, 0.9749760031700134, 1.095163106918335, -1.0325671434402466, 1.3956875801086426, -0.05106082931160927, -0.3722158670425415, 1.1547929048538208, 1.3573863506317139, 2.175333261489868, 0.6616005301475525, -1.868199348449707, 1.199903964996338, 0.7581894397735596, 0.9701839089393616, 0.4817255735397339, -0.27402418851852417, 1.2864418029785156, -0.11453056335449219, -0.9152936339378357, 0.317949116230011, 0.9368904829025269, 0.764450192451477, 0.7486632466316223, 0.30686846375465393, -1.536584734916687, -1.983284831047058, -0.2844140827655792, -0.5889130234718323, -0.7501082420349121, -1.5101468563079834, -0.2550906240940094, -1.1420468091964722, -0.9823992848396301, -1.3810793161392212, 0.05834363028407097, -0.1756189465522766, -0.5430437922477722, 0.4137287437915802, 0.784019947052002, -0.6052805781364441, 0.3775891959667206, -0.8379320502281189, 1.6661405563354492, 0.37518438696861267, 0.08502812683582306, -0.12313002347946167, -0.47199368476867676, 3.0406997203826904, -1.7178895473480225, 0.9697108268737793, 1.3631041049957275, 0.056229911744594574, -0.736899733543396, -0.37842851877212524, 0.680651843547821, -0.2923467457294464, -0.4153990149497986, -0.8421111702919006, -0.7595381736755371, 1.0469328165054321, -1.236944317817688, 0.02303660474717617, 0.3774343729019165, -0.4421095848083496, 0.15209521353244781, -0.1554744839668274, 0.5461689233779907, -0.9552006721496582, -1.0776922702789307, 0.30517446994781494, -0.4265427887439728, -0.5693385004997253, -1.1885696649551392, 1.555080533027649, 0.8022505044937134, 0.02479449473321438, 0.22258363664150238, -0.3190731704235077, -1.744271993637085, 0.7232245206832886, -0.24246518313884735, 0.6604387164115906, 0.9382363557815552, 0.4127471148967743, 0.5220897197723389, 0.4870453476905823, -0.5259173512458801, -0.09187135100364685, -0.5514033436775208, 0.9800915122032166, 0.4418518543243408, -0.10929475724697113, -0.23641669750213623, 0.7113268375396729, 1.118975043296814, -1.6296437978744507, -0.5820546746253967, -0.32876071333885193, 1.2441083192825317, -1.421549677848816, -0.19766542315483093, -0.07696933299303055, -0.7832868099212646, 0.03780288249254227, -0.5335537195205688, -0.6035770773887634, -0.47464942932128906, -0.08422669023275375, -0.9887312650680542, 0.09952563792467117, -0.8706567883491516, 0.31766173243522644, 2.1390230655670166, -0.2876494526863098, 1.3014955520629883, 1.3844090700149536, 0.06318184733390808, -0.9293460845947266, 1.0706928968429565, -1.9705290794372559, 0.7020094990730286, -1.0515094995498657, 1.151822805404663, -0.19907937943935394, 0.8257541656494141, -0.12693758308887482, -0.16058441996574402, -1.9138011932373047, 0.22870460152626038, -0.9184026718139648, 0.30789828300476074, 0.4362993836402893, 0.17743106186389923, 1.9382312297821045, -0.018336202949285507, -0.4227025508880615, -0.4079115390777588, -0.4348268508911133, 1.0980697870254517, 0.9436512589454651, -1.086429238319397, 0.4398137331008911, 0.0800764262676239, -1.988519310951233, 0.21389789879322052, 0.5831170678138733, 0.8773804903030396, 1.2213428020477295, 3.1773264408111572, 0.6030439138412476, 0.685619592666626, 0.8904297947883606, 0.4291016757488251, -1.1300151348114014, 0.8340597152709961, -0.02944863773882389, 2.8754894733428955, -2.2217607498168945, 1.059537410736084, 1.8436057567596436, 1.4960335493087769, -0.7704205513000488, -2.8219807147979736, -0.33482053875923157, -0.359861820936203, -0.2642524838447571, 2.4420464038848877, -1.073958396911621, 1.1810749769210815, -2.216367483139038, -0.6702948212623596, 0.15459206700325012, 1.6135330200195312, 1.2891197204589844, 0.21301835775375366, 1.1599880456924438, -2.1526126861572266, 0.8717195391654968, -0.7132308483123779, 0.8063752055168152, 1.6043083667755127, -0.6258237957954407, 1.0928068161010742, -0.5460379719734192, 0.8143931031227112, -1.1547338962554932, -0.6016160845756531, -0.16125717759132385, 0.42447444796562195, -0.7242885828018188, -1.375731110572815, 0.8996255397796631, -0.058706894516944885, 0.43450531363487244, 0.255950391292572, 1.4548616409301758, 0.13650181889533997, -0.4116170108318329, 1.4372607469558716, -0.15860043466091156, -0.035023413598537445, -1.6442406177520752, 0.7741086483001709, 0.4511924088001251, -0.9981294274330139, -0.8140219449996948, -0.23866970837116241, -0.05710655450820923, 0.36272382736206055, 1.7509135007858276, -0.19809085130691528, 1.1791033744812012, -0.5019152760505676, -0.33722805976867676, -0.24839361011981964, 0.3721766769886017, -0.0271442923694849, -0.9963555335998535, 0.7675989270210266, -1.138824462890625, 1.5299605131149292, -0.04259645938873291, -1.329580307006836, -1.972775936126709, 0.3701821267604828, -1.2812871932983398, 0.5121089816093445, 0.020389756187796593], [0.8744471669197083, 0.3126049041748047, 0.8866676092147827, -0.11220307648181915, -0.2149934321641922, -2.1739871501922607, -0.2131272405385971, 0.8796741366386414, -0.38955867290496826, 0.03655868396162987, -1.2461926937103271, 0.3486771583557129, 0.2188609391450882, -2.417762279510498, -1.3634381294250488, 1.2959489822387695, 0.31080135703086853, -0.3769896924495697, 0.3762115240097046, -0.40522733330726624, 0.76173996925354, 0.844390332698822, 1.4629982709884644, -0.6781675219535828, 0.18987104296684265, -0.20804770290851593, -0.679556667804718, 1.7235933542251587, 0.6975071430206299, -1.8077212572097778, 0.019490478560328484, -0.36748775839805603, -0.5277549624443054, -0.4729178845882416, 0.8547540903091431, 0.8422393798828125, -0.5071765184402466, -0.6198249459266663, -0.7827600836753845, 0.15151025354862213, 0.19916000962257385, 1.0339406728744507, 0.6883001327514648, -0.3225161135196686, -1.1802122592926025, -0.23033364117145538, 0.2029692679643631, 0.40094560384750366, 1.2334046363830566, -0.14413800835609436, -0.7508355379104614, -1.009740948677063, -1.3333978652954102, 0.5031641721725464, 0.09762713313102722, 0.4363434314727783, 2.094585657119751, 0.1452796906232834, 1.2478915452957153, -0.5099831223487854, 0.9468573331832886, 0.5078682899475098, 0.3419858515262604, -0.8215693831443787, 0.6412700414657593, 1.2387192249298096, -1.6791167259216309, -1.3643693923950195, 0.45539525151252747, -0.10232505202293396, 1.1021126508712769, 0.9464878439903259, -0.3317466378211975, -0.059338655322790146, 1.9699429273605347, 1.3954534530639648, 1.191942811012268, -1.7360107898712158, -0.14289075136184692, 0.004320899955928326, 0.5270969271659851, 0.6552611589431763, 0.9924964904785156, 1.6839392185211182, -1.609281063079834, 0.6223925948143005, -0.5615805983543396, -1.0153263807296753, 1.3764163255691528, -0.2546437382698059, -0.6070393919944763, -0.4089236557483673, -1.1134521961212158, -0.7795600891113281, -0.25325313210487366, 0.68283611536026, -0.05428947135806084, -0.16223061084747314, 0.2526474893093109, -0.8653541207313538, 2.1654367446899414, 0.3196313977241516, 0.9929991960525513, -0.31061476469039917, -0.9336640238761902, -0.2672617435455322, 0.21940165758132935, -0.6131651401519775, -2.490823984146118, -0.49417242407798767, 0.49446243047714233, 1.7527809143066406, 1.6337116956710815, -0.026012888178229332, 1.000862717628479, 1.3825135231018066, -1.1865874528884888, 0.02130793407559395, 2.1495602130889893, -1.5145833492279053, 0.5538308024406433, 1.1877198219299316, -0.9540420174598694, 0.5836902260780334, -0.555031955242157, 1.0886155366897583, 0.2204885184764862, 0.4440888464450836, -0.48282375931739807, 0.2485596388578415, 0.28814515471458435, 1.1063491106033325, 0.3899984657764435, 1.0488024950027466, 1.1812771558761597, 0.4554721415042877, -0.5561912059783936, -1.6657050848007202, 1.382543683052063, 0.7543068528175354, -0.19863732159137726, -0.7038930654525757, 0.6349248290061951, -0.2725287675857544, -0.24144497513771057, 1.0244874954223633, -0.32530641555786133, 0.36401623487472534, 0.8981693983078003, -1.7905914783477783, 0.5914027690887451, -0.4541471302509308, -0.28385233879089355, -1.2109519243240356, -1.198041558265686, 0.10959634929895401, -0.13290207087993622, 0.02752681076526642, -0.7797743678092957, -2.1784255504608154, 0.8261905908584595, -0.9867300987243652, -0.370353102684021, 0.48024627566337585, -0.43532130122184753, 1.657148003578186, -1.0291039943695068, -0.383672297000885, -1.4565337896347046, 0.29716262221336365, -0.7406263947486877, -0.14238348603248596, -1.3545911312103271, -0.4197360575199127, 0.7567697167396545, -0.05482349172234535, 1.264570951461792, 0.25896623730659485, -0.849785327911377, -0.8786059021949768, 0.9489098191261292, 0.6297773718833923, 1.173946738243103, -2.487501859664917, -0.10677863657474518, -1.3050519227981567, -1.5314898490905762, 0.8620395660400391, -0.4659375548362732, -0.9965827465057373, -1.5427942276000977, -0.8022946119308472, 0.0738716721534729, 0.01874890923500061, 1.4873287677764893, -0.9335484504699707, -0.4902615249156952, -0.5288357734680176, -0.9346736073493958, -0.038433756679296494, 0.283122181892395, 0.23442654311656952, -0.08485645800828934, -1.675338625907898, 0.15899749100208282, 0.7090666890144348, 0.4243355095386505, 3.1237633228302, 0.14761215448379517, 0.6454799771308899, -0.8490681052207947, 0.7640418410301208, -0.933362603187561, -0.7734619975090027, 0.9706993699073792, 0.4598865807056427, -0.9342771172523499, 0.1327747255563736, 0.3489447236061096, -0.1101582795381546, -2.3023829460144043, 0.4967135488986969, -0.07429566234350204, -0.9695438146591187, -0.8196606636047363, 1.4149419069290161, 0.07191481441259384, 1.5534965991973877, 0.4220484495162964, 0.26829075813293457, 0.6374666690826416, 0.3526786267757416, -2.694608211517334, 0.8489524126052856, 0.940605103969574, 1.178375005722046, 1.2375690937042236, -0.6600669622421265, -0.6172387599945068, -1.96942937374115, 0.07913582026958466, -0.1864355057477951, -0.6323595643043518, -1.6552530527114868, 0.6676225662231445, 0.15951815247535706, 0.4562900960445404, 0.5231837034225464, -1.4358915090560913, 0.7584691643714905, 1.2640318870544434, 0.313230037689209, 0.3056275248527527, -1.2095760107040405, 0.28703293204307556, 1.2982197999954224, -1.8644897937774658, 0.19186240434646606, -0.06658142060041428, 0.006353604607284069, -0.3019656836986542, 0.06738939136266708, -1.9140046834945679, 0.04767291247844696, -1.130733847618103, 1.0206503868103027, 0.6073877811431885, -1.4233556985855103, -1.5004608631134033, 0.9781122803688049, 2.2966039180755615, 0.10183535516262054, 2.527574062347412, 1.5643467903137207, 0.47947949171066284, -0.9956151843070984, 1.8293906450271606, -1.0345966815948486, 0.1361972987651825, -1.2541368007659912, 0.32320258021354675, 0.6363339424133301, 0.5315327644348145, 0.23650631308555603, -0.39568576216697693, -0.12945573031902313, 0.031746916472911835, 0.18751409649848938, 0.04620841518044472, -0.1931835114955902, -0.13200324773788452, 0.6808497309684753, 0.15456640720367432, 1.7287575006484985, -0.6298788189888, 1.099884033203125, 1.623308777809143, 0.22650253772735596, -0.17165504395961761, 0.37917736172676086, -0.160782128572464, -0.9049772620201111, -1.232986569404602, -0.16543161869049072, -0.4771302044391632, 0.48383283615112305, -0.14805184304714203, 1.8195546865463257, -0.154539093375206, 1.4083794355392456, -0.5539324879646301, -1.547289252281189, -0.3507199287414551, 0.19959332048892975, -0.5404053330421448, 1.091450810432434, -0.1051192507147789, 0.2184491902589798, -0.31570011377334595, -0.28251418471336365, -0.8074710965156555, 0.30704519152641296, -0.9693566560745239, -1.3819968700408936, -0.5059797763824463, 0.5657487511634827, -1.3248493671417236, 0.9689689874649048, -2.5067989826202393, 0.04122967645525932, -1.4277691841125488, -1.4580423831939697, -0.2905828356742859, -0.39015278220176697, 0.01980992779135704, 0.7358670830726624, -0.7181477546691895, 0.6918147206306458, -1.0201754570007324, -0.06464444100856781, 0.4923916757106781, -0.834328293800354, -1.9686760902404785, -0.3771430253982544, 0.1857755035161972, 0.5417895913124084, -1.7260771989822388, 0.5778862833976746, 1.4276236295700073, 0.5625237822532654, -0.10671891272068024, -1.5857502222061157, 1.013856053352356, 1.1521974802017212, 1.765924096107483, -0.7646273374557495, -1.1740556955337524, 0.9604411721229553, -0.35733357071876526, 0.9731101989746094, 0.19746644794940948, -0.23883303999900818, -0.8913269639015198, 0.21297000348567963, 0.5680760145187378, 0.8280443549156189, -1.3577044010162354, -0.9021857380867004, 1.0227863788604736, -1.9148308038711548, -1.3770854473114014, -0.1383594125509262, -0.32478824257850647, -1.7369626760482788, 0.37109753489494324, 0.5101757049560547, 1.422825574874878, -0.8801791071891785, 0.9935709834098816, 0.025465484708547592, -0.15266458690166473, 0.3989505171775818, -0.25880521535873413, 0.02673901617527008, -0.6945961713790894, -0.6030533909797668, 0.7046304941177368, 0.031520288437604904, 0.9064522385597229, -0.12255217134952545, -0.5190044045448303, 0.39649203419685364, -0.0547424852848053, 0.9672092795372009, 0.23493030667304993, -0.16088688373565674, 0.5394802093505859, -0.16132229566574097, -2.0679574012756348, -1.0602447986602783, 1.6187734603881836, 1.3491588830947876, -0.6471995711326599, -1.230304479598999, -0.494840532541275, -0.1057724580168724, -0.17720629274845123, -1.5291683673858643, 1.4585310220718384, -0.3980002701282501, 1.258774757385254, -0.4861604869365692, -0.414135217666626, 1.5070699453353882, 1.0066206455230713, 0.5659468173980713, -1.1472424268722534, -0.19828858971595764, 0.2539822459220886, -2.0123910903930664, 1.159708023071289, -0.8137701749801636, 0.17989955842494965, -0.032699160277843475, 1.4579777717590332, 1.6422791481018066, 0.34620240330696106, -0.11485428363084793, 0.05724925547838211, -1.0787791013717651, 1.2884773015975952, -0.5461655855178833, 0.47430410981178284, -0.016301684081554413, 1.2226369380950928, 0.3358016014099121, -0.8671145439147949, 1.528469443321228, -1.2653902769088745, -1.1678427457809448, 0.6734533905982971, -0.8389328718185425, -0.08679555356502533, 0.43965449929237366, -0.6580942273139954, 1.355543613433838, 0.5781316757202148, 0.13169750571250916, -1.9284695386886597, 0.04465309903025627, -0.015334044583141804, 0.3936098515987396, 0.030410945415496826, -0.9583816528320312, -0.010916698724031448, -0.632411539554596, -1.1660040616989136, -0.5654580593109131, -0.9304459691047668, -0.2142818719148636, -0.1649722307920456, -0.07692521065473557, 1.3837159872055054, -0.8362791538238525, 0.06175932288169861, 0.8666011691093445, 0.46567991375923157, 0.5435643196105957, -0.34865352511405945, -2.3744957447052, 1.9548956155776978, -1.199993371963501, 0.9000703692436218, 0.41543498635292053, 0.749571681022644, -0.8237667679786682, -0.7259978652000427, 2.1525425910949707, 0.7378596067428589, -0.8024045825004578, 0.8504735827445984, 0.6344292163848877, 0.4498711824417114, -0.9333739876747131, 0.6401491761207581, 1.1552139520645142, 0.41542986035346985, 0.9874237775802612, 2.745891809463501, -1.4857492446899414, 0.3573324978351593, -0.6709505319595337, 0.26251617074012756, -0.927585244178772, 0.9231720566749573, -1.1657435894012451, 0.7886063456535339, 1.3260698318481445, 0.42808040976524353, 0.687499463558197, -1.454635500907898, 1.5820388793945312, 0.600002646446228, -0.6483619809150696, -2.3345413208007812, -2.260181427001953, 1.3775184154510498, 0.0937291607260704, -0.06131858006119728, 0.5647950172424316, -0.18154190480709076, -0.7536292672157288, 0.13702256977558136, -0.33141565322875977, -1.5008127689361572, 0.3514805734157562, -0.8401100635528564, 0.5428864359855652, 0.4325423836708069, 0.4950546324253082, -1.3185540437698364, -1.1887527704238892, -0.942425549030304, -1.9958701133728027, 1.2577793598175049, -0.601722002029419, -1.2054301500320435, 1.2188924551010132, 0.8201860785484314, -0.6005164980888367, 1.2325669527053833, -0.17631439864635468, -1.7478560209274292, 0.26120781898498535, 0.6206827759742737, 0.04562360793352127, 0.308206170797348, 0.751056432723999, -0.33650532364845276, -0.9293110966682434, -1.8306444883346558, -0.37136343121528625, 0.682781994342804, -0.7220346331596375, -1.0425893068313599, 1.1394054889678955, -1.336824655532837, 0.7629228830337524, 0.6047177910804749, -0.762586236000061, 0.4473961591720581, -0.35284224152565, -0.3441123366355896, 1.1753160953521729, -1.1942322254180908, 0.09849165380001068, -0.14103296399116516, -2.5655596256256104, 1.086998701095581, 0.8344240188598633, -2.7967677116394043, -0.3675607144832611, 0.3485954701900482, 0.278696209192276, 0.5369985103607178, -0.236544668674469, -0.4919520318508148, -0.5565954446792603, -0.49866291880607605, -0.07449860125780106, 0.5742816925048828, 2.7698543071746826, 0.11230103671550751, -0.06102846935391426, 1.3658790588378906, 1.3331512212753296, 0.4780653417110443, 1.838576078414917, 0.032681070268154144, 0.3891749083995819, 0.5884425640106201, 0.7787397503852844, 0.4154060184955597, 0.37551888823509216, -0.4623279869556427, 1.1073719263076782, -0.4363480806350708, -0.6813386678695679, 0.5564996600151062, 0.4764772653579712, 1.3119001388549805, -0.3676873743534088, -1.79373300075531, -1.4438011646270752, 2.7703192234039307, -1.002030372619629, -0.15875889360904694, 0.41839373111724854, 0.21274332702159882, 0.7250184416770935, -1.7299258708953857, 0.06776875257492065, 0.3111273944377899, 0.6677613258361816, -0.15077728033065796, -0.4037140905857086, -1.957376480102539, 0.7602430582046509, 1.2112765312194824, -0.4139764904975891, -1.560904860496521, 1.5057387351989746, 0.4142049252986908, -1.0417098999023438, 0.6276123523712158, -1.6025694608688354, 0.5195913910865784, 0.4909389019012451, 0.13283996284008026, -1.4147751331329346, -0.393015056848526, -0.5559526681900024, 0.40625321865081787, 1.5165132284164429, -0.9130739569664001, 0.8621137738227844, 1.9182467460632324, -0.071412593126297, -2.4748876094818115, 0.8098310232162476, 1.1578701734542847, 1.8260118961334229, 1.3229066133499146, -0.09630720317363739, 0.7441482543945312, -1.4738843441009521, 0.4893788993358612, 0.0554073229432106, 0.43471333384513855, 2.250199794769287, 1.0945870876312256, 0.9525320529937744, 0.2637656033039093, -2.1221723556518555, 1.220170497894287, -0.7095361351966858, 0.038542088121175766, -0.7323653101921082, 0.3791466951370239, -0.5754179358482361, -1.5269739627838135, -0.6223273873329163, -0.9854607582092285, 0.6719952821731567, 0.9810869693756104, -1.232540249824524, 0.7498028874397278, -0.12995105981826782, -0.36271339654922485, 0.9910261631011963, -1.2061635255813599, -0.01824582926928997, 0.17295460402965546, 0.3563947081565857, 0.8226218819618225, 0.2778013348579407, -0.004849683493375778, 1.499244213104248, 0.31212761998176575, 1.060360074043274, 1.0451208353042603, -1.2238420248031616, 0.09296292066574097, -0.32083284854888916, -1.1682952642440796, 2.552119016647339, 0.5394774079322815, -1.9829521179199219, -0.41202521324157715, 1.0083595514297485, 0.07818382978439331, -0.7355955243110657, 1.7902864217758179, 1.218652367591858, -0.8800703287124634, 1.8585747480392456, -0.9872584342956543, 1.004941701889038, 0.6308804750442505, -0.7663770914077759, 0.04282589629292488, -1.2052556276321411, 2.430539131164551, 0.41029441356658936, -0.7165126204490662, -1.1957669258117676, -1.2254652976989746, 1.4207038879394531, -0.16109076142311096, -0.9407048225402832, 0.4963170289993286, 0.12122916430234909, 1.7596547603607178, 1.7467397451400757, 0.896470844745636, -0.4660305976867676, -0.08239291608333588, -0.32566002011299133, 0.7069205641746521, 0.5746870636940002, -0.6077557802200317, 1.220197319984436, -0.6395236849784851, -0.08064234256744385, -1.6125584840774536, 1.48940110206604, -0.5920109152793884, 1.5131285190582275, -0.9453742504119873, 0.2419077306985855, -0.00371619313955307, 1.3063749074935913, -1.5622071027755737, 0.34941366314888, -0.4280852675437927, 0.0835072323679924, -0.5985901355743408, 0.26411762833595276, 1.9012582302093506, -1.970759391784668, -0.3803027868270874, 0.21441714465618134, 1.4229450225830078, 1.1345278024673462, 0.44093233346939087, -0.45194682478904724, 1.6761925220489502, -0.8014559745788574, 0.577553927898407, 1.0728317499160767, 0.24915526807308197, 0.7380242347717285, -0.28721535205841064, 1.5446605682373047, -1.877327799797058, 0.003288572421297431, -1.4830303192138672, -0.19119668006896973, 0.40034133195877075, -1.2765858173370361, -0.10772339999675751, -0.45603758096694946, -0.693984866142273, 0.048567403107881546, 0.01642998866736889, 0.5626974105834961, -1.3982614278793335, 1.6631848812103271, -0.14676061272621155, 0.24279797077178955, -0.06788836419582367, 1.4593459367752075, -0.15067318081855774, 0.7403714060783386, -1.691640019416809, 0.8883633017539978, -0.6028409004211426, 0.22319859266281128, -0.33501875400543213, 2.3220760822296143, 0.22782155871391296, 0.5534485578536987, -0.5348497033119202, 0.9077533483505249, -1.1225332021713257, -1.4092609882354736, 0.5326358079910278, 0.039518631994724274, -0.27395308017730713, -1.4239915609359741, -1.2790533304214478, 0.9934875965118408, -0.8748217225074768, -0.06618716567754745, -0.22113221883773804, -1.7119171619415283, -0.624279797077179, 1.0709867477416992, -0.7192884683609009, 0.14083832502365112, 0.9387531876564026, 1.9504563808441162, -1.841623306274414, -1.7177423238754272, -0.7817687392234802, -3.205613851547241, -1.2137788534164429, 1.294919729232788, 0.6526871919631958, -0.358797162771225, 0.14908979833126068, 0.9679314494132996, 0.9599042534828186, -1.1726734638214111, -1.4933407306671143, -1.3859639167785645, 0.6256313323974609, -0.11126109212636948, 0.361238032579422, 0.1506509780883789, -1.3263280391693115, -0.4845147728919983, -0.44774287939071655, -2.0746233463287354, -1.3499828577041626, 0.4719059467315674, 0.26553693413734436, 0.06341908127069473, -0.8319841027259827, -0.30182942748069763, -0.1525631695985794, -0.202516108751297, -1.8747692108154297, -0.040804486721754074, -0.7612356543540955, 1.3068962097167969, -0.5271490812301636, -1.8482528924942017, -0.5116443037986755, -0.22069048881530762, -0.34486427903175354, -0.7352098822593689, 1.0019944906234741, 1.0032333135604858, 1.8597517013549805, -1.9420204162597656, -0.4325907826423645, -0.5164386630058289, 0.5571258664131165, 0.23128542304039001, -0.4233187735080719, -1.5869255065917969, -0.38919147849082947, -0.810232400894165, -0.5650753974914551, 0.3851712644100189, -0.7548869848251343, -0.42102691531181335, -0.32199105620384216, -0.9627301692962646, 0.9733464121818542, -0.5215147137641907, 0.32311633229255676, -2.1131205558776855, 1.148311734199524, 0.6923290491104126, 0.30179503560066223, -0.7079684734344482, -0.035710930824279785, -0.7572231292724609, -1.5669479370117188, 1.1693298816680908, 1.5596213340759277, -0.7422260642051697, 0.017582859843969345, -0.5597662925720215, -0.49409833550453186, 0.6362136006355286, 0.9808349013328552, 0.9970484972000122, 0.6533146500587463, -1.0416496992111206, 0.7648317813873291, 2.985717535018921, -0.8159857392311096, 0.5709648728370667, -1.2758194208145142, -0.8256964087486267, 2.0984697341918945, 0.6609652042388916, -0.12183154374361038, -0.6830807328224182, 1.7214831113815308, -0.10810541361570358, 0.17640803754329681, -0.1526649296283722, 0.6142966747283936, 1.0089141130447388, 1.0827056169509888, 0.8807953596115112, -0.6806625723838806, 0.5428781509399414, -0.6460021138191223, -0.358270525932312, -0.3019510805606842, -1.821639895439148, 0.03659651055932045, -0.5736867785453796, 2.0304648876190186, 0.40190428495407104, -0.8197318911552429, 0.38598692417144775, -0.8773638606071472, 0.3658692240715027, 1.0479352474212646, 0.12098882347345352, 0.07201030850410461, -0.03028816729784012, -0.3869777321815491, -0.6129570603370667, 0.2584303617477417, 0.428525447845459, -0.6580473184585571, -0.6938285827636719, 0.7093998193740845, 1.6518731117248535, -0.1638375222682953, -1.4200247526168823, -1.3981728553771973, 1.398152232170105, 1.2679080963134766, -0.16126130521297455, -1.0321515798568726, 2.1154017448425293, 0.06553308665752411, -1.524476170539856, 0.21672266721725464, 0.08949151635169983, -0.1706441193819046, -1.207066535949707, 0.10047615319490433, 1.881593108177185, -0.41625285148620605, 0.090330570936203, -0.5568324327468872, -1.7256202697753906, 0.13856522738933563, 0.5471706986427307, 0.20745377242565155, -0.24202734231948853, 0.32161128520965576, -0.0655263289809227, -1.224851131439209, 1.0121827125549316, -0.2038668990135193, 2.332765579223633, -0.1340930014848709, 1.071230411529541, 1.270107388496399, -0.27771925926208496, 1.0035583972930908, 1.1910531520843506, -1.11408531665802, -0.16281110048294067, -0.19583524763584137, 0.1937568187713623, -0.3233449161052704, -0.24755889177322388, 1.2907973527908325, 1.8270213603973389, 0.3952260911464691, 0.8652570247650146, -1.6308965682983398, 1.4503809213638306, 0.05961336940526962, 2.17587947845459, -0.27561718225479126, 1.299228310585022, 0.9756876826286316, -1.8132672309875488, -1.138520359992981, -0.24068467319011688, 0.528497576713562, 1.3828039169311523, 0.3035041391849518, -1.5350559949874878, 0.8580204844474792, 0.6122293472290039, -0.21662898361682892, 0.7360762357711792, -0.07883644104003906, 0.6789446473121643, 0.8152410984039307, 1.659922480583191, -0.4225353002548218, -0.9389687776565552, -1.673919677734375, -0.4785176217556, -0.361137717962265, 0.3218698799610138, -0.35265907645225525, 0.5468406081199646, 0.4118348956108093, 0.006845328025519848, -0.5400675535202026, -0.3958420157432556, -0.16744618117809296, 1.234411597251892, -1.8454031944274902, -0.3826751410961151, -0.5397579073905945, -0.22535720467567444, -0.27483895421028137, -0.7144936919212341, 0.3837003707885742, 0.1570456624031067, -1.634660005569458, 0.9136396050453186, -0.29808852076530457, -0.3223704695701599, -0.8686246871948242, -1.133154034614563, -0.11867019534111023, -1.197898030281067, -0.23499695956707, 0.14256545901298523, -1.6774605512619019, 1.2712085247039795, -1.020382046699524, 0.9991331100463867, 0.07463997602462769, -1.033257007598877, -0.503888726234436, -0.49436837434768677], [1.390619158744812, -0.9866277575492859, -0.5942530035972595, 0.4268858730792999, -1.0628622770309448, -1.4859914779663086, 0.7105099558830261, 0.776859700679779, 1.3731763362884521, -0.04694839194417, 0.020770065486431122, 0.004848142620176077, -1.9264683723449707, 0.5655934810638428, 1.3544665575027466, -0.5836697816848755, 0.20251308381557465, -0.4607010781764984, 0.37255728244781494, 0.5069666504859924, 0.1495390385389328, -0.7190215587615967, -1.8083442449569702, 0.30214664340019226, 0.6241541504859924, -0.04926597326993942, 2.2142224311828613, -0.9739232063293457, 0.4330330193042755, -0.1906662881374359, -1.3887766599655151, -0.6601598262786865, -1.0831395387649536, 0.6616370677947998, -0.7825666069984436, 0.5071530342102051, 0.43541017174720764, -0.47698402404785156, 0.313711941242218, -0.8630850911140442, 1.865631341934204, -0.43251121044158936, -0.6013563871383667, -0.9849388003349304, 0.0045580510050058365, 1.9400444030761719, -2.338489294052124, -1.9648665189743042, 0.705635666847229, 0.31925761699676514, -0.23601804673671722, -0.8728395700454712, -0.5303328633308411, -1.447105884552002, -0.0474255308508873, -0.21863581240177155, 0.34865257143974304, -0.4678959250450134, -0.29811891913414, -1.4169665575027466, 0.21911892294883728, 0.35450389981269836, 0.21143381297588348, -0.6822143793106079, -0.7670671343803406, 0.10770159214735031, -0.32778289914131165, 0.22664661705493927, -0.764030396938324, -1.2229009866714478, 1.364725947380066, -1.026708722114563, 0.5456159710884094, -0.06382188946008682, -1.0895086526870728, -1.0038723945617676, -0.8990798592567444, -1.096293568611145, 0.6170006394386292, -0.031654421240091324, -0.060331348329782486, -0.8197057247161865, -0.22871582210063934, -0.43501338362693787, 1.270068645477295, 0.7152446508407593, -0.5813910961151123, 1.9889129400253296, -2.2311532497406006, -1.7029067277908325, 0.731597900390625, -0.17646755278110504, -2.734221935272217, 0.5734374523162842, -0.5645120739936829, 1.5051966905593872, 0.4255147874355316, -0.08094336092472076, -0.7291057705879211, 0.9389773607254028, 0.8560089468955994, 0.21628494560718536, 0.42796456813812256, 0.02555091492831707, -1.6343052387237549, 0.05541867017745972, -0.7229755520820618, 0.35321155190467834, -0.22836123406887054, 0.050489798188209534, -0.2022046148777008, 0.5238624215126038, 0.4323614835739136, -0.27256348729133606, -0.13221469521522522, -0.36006277799606323, -1.4529423713684082, -0.32422107458114624, -0.3065261244773865, 0.3572787642478943, 0.06670204550027847, -0.2906932830810547, -0.7283649444580078, -0.4649491608142853, 0.9088245630264282, -1.1619815826416016, 0.211070716381073, -0.5889652371406555, -1.505784273147583, -1.9380123615264893, -0.3860562741756439, -0.8806983232498169, 1.0337564945220947, 0.5331158638000488, 0.9722831845283508, 0.4996272027492523, 1.3928732872009277, -0.4733501672744751, -0.6776381731033325, -1.215901255607605, -0.08467363566160202, -0.3711335361003876, 0.2709164619445801, 1.9728018045425415, -1.317468523979187, 1.4940704107284546, 0.5382668972015381, 0.579412579536438, 1.3191388845443726, 0.008018199354410172, -0.9197396636009216, 0.7922266721725464, -0.3671759068965912, -0.3915416896343231, 0.5215765833854675, -0.07597963511943817, 1.8704105615615845, -1.8950247764587402, -1.0851165056228638, 1.5398461818695068, 0.5762781500816345, -0.859734058380127, -0.8340596556663513, 0.047680068761110306, -0.755723774433136, -1.3786858320236206, 0.21639250218868256, -0.31591159105300903, 0.16621096432209015, 0.5882704854011536, 0.6584421396255493, 1.6079235076904297, -0.7856168746948242, 0.026996543630957603, -0.8354417085647583, -1.5548555850982666, -0.696239173412323, -1.9042927026748657, -1.4202805757522583, -0.04882952198386192, -0.7468141317367554, -0.26316097378730774, -1.5301272869110107, -1.330570101737976, -0.7587204575538635, -1.7823466062545776, -0.9384031295776367, -0.08888442069292068, -1.917559027671814, 0.8988269567489624, 0.47377312183380127, -0.5423504710197449, -0.4510921239852905, -0.5816202163696289, 1.6540157794952393, -1.0779168605804443, -1.7410252094268799, -1.654687523841858, 0.018179282546043396, -1.9494761228561401, -1.5482559204101562, -0.7554796934127808, 1.0569508075714111, 0.7880553603172302, 0.5705801248550415, 1.1598981618881226, -0.9789323210716248, 0.49303823709487915, 0.52970290184021, 0.09634854644536972, -0.4921189546585083, 2.3157458305358887, 0.014195098541676998, -0.63877934217453, 0.03927164897322655, 1.207509994506836, -0.29945483803749084, -1.1048052310943604, -0.7703996300697327, -0.7843762040138245, -1.8410208225250244, -0.8824297785758972, 0.5946739912033081, -1.2362933158874512, 0.4574402868747711, -1.5990406274795532, -1.7571226358413696, -0.5560632944107056, -0.14227919280529022, 0.3895507752895355, -0.16367016732692719, -2.4055917263031006, 0.615346372127533, -0.2821172773838043, -1.607948660850525, -1.4405813217163086, 0.5337987542152405, -0.14681202173233032, 0.0933350995182991, 0.5408236980438232, 0.5802035927772522, -0.1262338012456894, 1.912084698677063, -0.5068572163581848, 0.7172518372535706, 0.6365213394165039, 0.07218069583177567, -0.22430051863193512, -0.13464568555355072, -1.8329733610153198, 0.29282358288764954, 0.6334513425827026, -0.15357638895511627, 0.04443792253732681, -0.2988796532154083, -0.056872572749853134, -1.0411384105682373, 0.5801332592964172, -1.732814908027649, -0.7048128247261047, 0.8202679753303528, 1.063232183456421, 0.9281754493713379, 0.39685291051864624, 0.3741912841796875, 0.16764631867408752, 0.927094578742981, -0.5039033889770508, 1.6196866035461426, 1.7854194641113281, 0.4971076548099518, -0.21349796652793884, 1.6499927043914795, 2.1991302967071533, -0.9406841397285461, 0.7648091912269592, 0.45210766792297363, 0.16715942323207855, 0.31526362895965576, 0.6048300862312317, -0.6477739810943604, 1.683272123336792, 0.09988798946142197, -1.996935486793518, -0.6205369830131531, 1.0799038410186768, 0.014015477150678635, -0.6404491066932678, -0.6234039664268494, -0.4343256652355194, -1.825738787651062, 0.03097592107951641, 0.7301265597343445, 0.0048238798044621944, 0.2502431273460388, 0.42623379826545715, 0.511635422706604, 0.570231020450592, 0.8173298239707947, 0.8225390315055847, -1.7251063585281372, -0.3092936873435974, 1.2345659732818604, -0.0167009849101305, 0.9809395670890808, -0.11376316845417023, -0.10456399619579315, 0.5706703066825867, 0.5911679863929749, -0.7880730628967285, 0.34064981341362, -0.8781383037567139, -0.0999358668923378, -0.7759366631507874, -0.7228595018386841, 0.1755417287349701, -1.1638985872268677, -0.9941245317459106, 0.3942856788635254, -1.1663854122161865, -0.29644039273262024, 1.1561377048492432, 1.3612661361694336, -0.12180031090974808, 0.0007709171622991562, 2.0506210327148438, 1.1683586835861206, -1.1580148935317993, -1.4874813556671143, 1.496511459350586, 0.24225609004497528, 1.4116508960723877, -0.9238118529319763, -0.3865508735179901, -0.06572069227695465, -0.13840019702911377, 1.2599973678588867, 0.45356228947639465, -0.2073294222354889, 0.42784836888313293, -0.04934323579072952, 1.4756289720535278, 0.7526895999908447, -0.48759278655052185, -0.15489372611045837, -0.08135025203227997, -0.5044260025024414, -0.1301649510860443, 0.5680034160614014, -0.16230449080467224, 1.282850980758667, -0.9180818200111389, -0.05400697514414787, -0.05657759681344032, -1.2989786863327026, 0.9338719844818115, -0.7375912070274353, -0.9337413907051086, 1.7362439632415771, -0.46532779932022095, -0.010203080251812935, -0.20051762461662292, 0.031469106674194336, 0.7877235412597656, -0.6839653849601746, -0.21260066330432892, 1.5276894569396973, 0.7434076070785522, 0.35459190607070923, 0.10801935940980911, 1.5655553340911865, -0.6117615699768066, 0.696460485458374, 0.8280700445175171, -0.9430018663406372, -0.09965265542268753, 0.5644862651824951, -0.5889726281166077, 0.46689140796661377, 0.2957670986652374, -1.3233425617218018, 0.752480685710907, 1.1408302783966064, -0.8936628699302673, 0.6438746452331543, -0.026045778766274452, -1.4470354318618774, 0.3654717206954956, -1.0296683311462402, 2.7921576499938965, 0.6051797866821289, -0.31018033623695374, -1.5623576641082764, 0.40027573704719543, 2.2702224254608154, -0.8447165489196777, 0.09432706236839294, 0.19014517962932587, 0.23477791249752045, -0.7806047201156616, 0.7133470773696899, -0.6002830862998962, -1.3791054487228394, -0.9606568813323975, 0.33034247159957886, 0.5278083086013794, 0.23341040313243866, 0.3495388329029083, 0.023081814870238304, 1.265882968902588, -0.4479103684425354, -1.5651986598968506, -0.7520205974578857, -0.7591283917427063, -1.361146330833435, 0.20418496429920197, 1.2610899209976196, -1.2298204898834229, -0.3458768427371979, -0.14362037181854248, -1.3699367046356201, -0.3022398352622986, 0.2239728569984436, -1.099597454071045, -1.002243161201477, 0.7018188834190369, -0.7463899850845337, -0.8266751766204834, 0.820595383644104, -0.037345919758081436, 1.0363445281982422, 0.6186242699623108, 0.018131140619516373, 1.0268858671188354, -1.2980105876922607, 0.21745119988918304, -0.9654735922813416, 0.4522053599357605, 0.09577665477991104, 0.4710424244403839, 2.0312371253967285, 1.1327885389328003, 0.17869128286838531, -0.15284797549247742, -0.035254206508398056, 0.4598905146121979, -0.7877431511878967, 0.1673126518726349, 1.6354262828826904, -1.326991081237793, -0.19380226731300354, 0.08306471258401871, -0.04077858477830887, -0.8132163882255554, 0.5076890587806702, 0.5700061917304993, 0.28420528769493103, 1.1922640800476074, -0.7284753322601318, 1.707299828529358, -0.2439398318529129, 0.9599214792251587, -0.13013432919979095, 0.34203189611434937, -0.6395490169525146, 0.2300669550895691, 1.4195945262908936, 1.2218728065490723, 0.6481558680534363, -0.28783875703811646, 0.41816920042037964, -1.123740553855896, 0.40647459030151367, 0.5596402287483215, -1.801583170890808, -0.24640430510044098, -1.678311824798584, -0.3262770175933838, -1.2222084999084473, 0.6574296355247498, -0.08385671675205231, -0.2414720058441162, -0.36796921491622925, -1.195780873298645, -0.3819938898086548, -1.55519700050354, 0.4690164625644684, -0.48845139145851135, 0.7982436418533325, -0.7839393615722656, -0.45643338561058044, -1.2200316190719604, -1.505975365638733, 0.4680441617965698, 0.8750729560852051, -1.0310009717941284, -0.9896826148033142, 0.26717668771743774, 0.1531728059053421, 0.4201800525188446, 0.03455430641770363, 0.032355327159166336, -1.583259105682373, 1.191225528717041, -1.073746919631958, 0.5397878289222717, -0.32074615359306335, -1.4617116451263428, 0.08297872543334961, 0.11243563145399094, 0.2802621126174927, -0.7980771064758301, 0.5990447998046875, 0.3662736415863037, -1.1888797283172607, -0.07408960163593292, 2.4806156158447266, 0.30771404504776, -1.0521442890167236, -1.2004936933517456, -0.02378329075872898, -0.4537115693092346, -1.9910801649093628, 0.15863028168678284, 1.0299516916275024, 0.18740640580654144, 1.1294636726379395, -1.4675346612930298, -0.7127670645713806, -1.1322684288024902, 0.09396178275346756, -0.2686872184276581, 1.0035992860794067, -1.4256517887115479, 0.004186692181974649, 1.9373061656951904, 0.15068680047988892, 1.1848399639129639, 0.011835568584501743, 0.0028094409499317408, -2.1488609313964844, 0.6745793223381042, 0.9353405833244324, 0.99582839012146, -0.7415023446083069, -0.8962700963020325, -0.030294928699731827, -1.739669680595398, -1.0156465768814087, 1.4952867031097412, 0.12518525123596191, 1.0685393810272217, 0.48763900995254517, -0.4918634295463562, -0.9982987642288208, 0.8908174633979797, 0.6881729960441589, 0.6318227052688599, -1.5410081148147583, 2.2395477294921875, -0.3065956234931946, -0.42010337114334106, 0.9215457439422607, -0.45018747448921204, 1.3218040466308594, -1.19385826587677, -0.13937707245349884, 0.8771246075630188, -0.6410306692123413, 1.8419780731201172, -0.8722108602523804, 2.1874217987060547, 1.1119186878204346, -0.14599578082561493, 0.3144719898700714, 2.1250100135803223, -0.45280274748802185, 0.0408964529633522, 0.35840800404548645, 0.11323004215955734, 0.33403313159942627, -0.1615193486213684, -1.851763367652893, -0.6005762815475464, -0.16275961697101593, -0.2484482228755951, -0.6030609011650085, -0.9385026097297668, 0.6678203344345093, 0.050204962491989136, -0.37115973234176636, 1.7924846410751343, -1.9364509582519531, -1.916872501373291, 0.5521106123924255, 2.686659097671509, -0.702614963054657, 0.22783654928207397, 0.5736384391784668, 0.25067374110221863, 0.1105254516005516, -1.6623111963272095, -0.3411426544189453, 0.1745108813047409, 0.2786606252193451, 1.3714019060134888, 0.08588797599077225, 0.6711074113845825, 0.30331721901893616, 0.6216810941696167, -1.3889371156692505, -1.2834769487380981, -2.044910192489624, 0.07398566603660583, -0.5145273208618164, 0.7681906819343567, 0.9522579312324524, 0.5884971618652344, 1.3499908447265625, -0.16785207390785217, -0.030637603253126144, -0.011414248496294022, -1.1094141006469727, -1.6955511569976807, -0.4178919792175293, 0.356854110956192, 0.9806122183799744, 0.07979460805654526, -0.3357466459274292, -0.9903125166893005, 0.445544570684433, 0.6479076147079468, -1.0471631288528442, -1.0342406034469604, -0.27974823117256165, 1.2697707414627075, 0.5297945141792297, 0.057832423597574234, 0.32800740003585815, -0.9001758098602295, 1.0914015769958496, 1.1215991973876953, 1.3049092292785645, -0.9567609429359436, -1.5547661781311035, -0.19122514128684998, 1.8606104850769043, 1.387516975402832, -0.6981712579727173, -0.20265643298625946, -1.8751049041748047, 0.9968926310539246, -1.460646629333496, -0.6214295029640198, 0.6956391334533691, -0.3728818893432617, 0.49636322259902954, 0.48216748237609863, -1.3244065046310425, 0.898489236831665, 0.12801219522953033, -0.5561209321022034, -1.3422086238861084, 0.2704814374446869, -1.0761624574661255, 0.21704944968223572, -0.9088647961616516, 0.8074063658714294, -0.4979555904865265, -1.2496709823608398, 0.8619349002838135, -1.0446170568466187, -0.08542367815971375, 0.2899109423160553, 0.3860839605331421, -0.04152612015604973, -0.4093385934829712, 0.7405704855918884, -1.0630987882614136, -1.3216239213943481, -0.7328482866287231, 0.47118043899536133, -0.28253859281539917, 0.3863617479801178, -0.24979686737060547, 0.013839188031852245, -1.5198636054992676, -0.7554019689559937, -0.506630003452301, -0.5565759539604187, 1.3114686012268066, 0.24607111513614655, -0.8238294124603271, -1.3313302993774414, 2.0153470039367676, 0.029193945229053497, 0.7464495301246643, -0.20663201808929443, -2.146294355392456, -0.3056327998638153, 0.20362770557403564, -0.7153839468955994, 0.010574951767921448, 0.9204409122467041, 0.012050607241690159, -0.8011816740036011, -0.6895541548728943, -1.2775864601135254, -2.862881898880005, -2.2288453578948975, 0.1035371944308281, 0.12180672585964203, -1.7309489250183105, 0.5166016817092896, -1.020819067955017, -3.155076026916504, 1.2508069276809692, 0.7316592335700989, -2.310837984085083, -0.9100183248519897, -0.8054348230361938, -0.13650216162204742, -0.03154584765434265, 0.08562741428613663, -0.144382044672966, -0.5107797384262085, -0.6871079802513123, 0.0207833219319582, 0.45782920718193054, 0.8389367461204529, -0.13943563401699066, -0.33610379695892334, -0.14168117940425873, 1.1207020282745361, -1.295246958732605, -1.074957251548767, 1.0970356464385986, -1.1769418716430664, -0.7142683863639832, -0.08106594532728195, -0.2786591053009033, -0.6698161363601685, -2.1358351707458496, -0.31504711508750916, -0.3038915991783142, 1.6098172664642334, -1.5240705013275146, 0.4414425492286682, -0.36359161138534546, -0.7111685872077942, 0.7557172179222107, 0.7335084676742554, -1.3464078903198242, 1.0578516721725464, -0.7617865800857544, 0.8243303894996643, -0.506554126739502, -0.12037529796361923, 0.1103694960474968, 0.8289377093315125, -1.4458847045898438, -0.017740523442626, 1.5619003772735596, -1.9061590433120728, 0.8768633604049683, 1.6310603618621826, 0.902861475944519, -1.453019142150879, -1.0698442459106445, -0.20171725749969482, -0.3840804696083069, -0.18062710762023926, 1.0294510126113892, 1.9586763381958008, 1.3235743045806885, -0.9638415575027466, 0.135639950633049, -0.07817155867815018, -1.0566773414611816, -1.2103681564331055, 0.04277021810412407, -1.848176121711731, 1.49387788772583, 0.5527715682983398, 2.4370973110198975, 0.2053416669368744, -0.30999481678009033, 0.026628069579601288, 1.2869559526443481, 0.41498374938964844, -0.8580152988433838, 0.4808494746685028, -0.24714885652065277, -2.385349750518799, 0.01314022671431303, 1.188842535018921, 1.054999828338623, 0.8049275279045105, 0.8601628541946411, -1.2473852634429932, -0.7588316202163696, 0.025512713938951492, -0.8278796672821045, -0.6436983346939087, 1.769461989402771, 0.4456750154495239, -0.25814875960350037, -0.05588066205382347, 0.20345905423164368, -0.4716619849205017, 1.9464904069900513, -0.20400318503379822, 1.9440064430236816, -0.6877670288085938, -0.1744542270898819, -0.391091525554657, -1.8025110960006714, -1.3946466445922852, 0.5635913610458374, 0.6864156126976013, 0.5919352173805237, 0.76662677526474, 0.45799529552459717, 1.1488808393478394, 0.3804137706756592, -0.16906504333019257, -0.27568110823631287, 0.9103623032569885, 0.04836928844451904, -1.1033927202224731, 1.1639859676361084, 1.1478939056396484, 0.7475003004074097, 0.002785980934277177, 0.5409284830093384, -1.0284466743469238, 1.3932870626449585, -0.16380982100963593, -0.34365537762641907, -0.7600789666175842, -0.3494004011154175, -0.14751692116260529, -1.3055487871170044, 0.20899759232997894, 0.5327101945877075, -1.4810982942581177, -0.8341781497001648, -0.7903971672058105, -0.683677077293396, -0.3137844502925873, 1.5737501382827759, 1.1869099140167236, -0.6163378357887268, 0.41209664940834045, 0.34507477283477783, -1.5431281328201294, -0.5126074552536011, 0.3260739743709564, -0.6624261736869812, -0.16577380895614624, 1.3340710401535034, -0.0705491453409195, 0.5938039422035217, 0.10179321467876434, -2.130549669265747, 0.6884558200836182, -2.123847007751465, -0.29921087622642517, 0.6483858227729797, 0.39285463094711304, 1.7333664894104004, -1.0011154413223267, 1.4400367736816406, 2.0429139137268066, -0.10165659338235855, -0.5501303672790527, -0.7011052370071411, 0.25103023648262024, -0.1507139950990677, -0.6697962880134583, 0.7534168362617493, 1.0497380495071411, 1.2075029611587524, 0.9438916444778442, 0.47547444701194763, -0.9499718546867371, 1.5540488958358765, 0.6554024815559387, 0.1593262106180191, -1.0460779666900635, -0.8166061639785767, 0.8152778744697571, -1.2194154262542725, 1.7012850046157837, 0.2466122806072235, -0.17115889489650726, -1.1265560388565063, 0.34897005558013916, 0.6088424324989319, 0.6181175708770752, 0.011298039928078651, -0.552961528301239, 0.15681061148643494, 0.8176131844520569, 0.9774781465530396, -1.6590464115142822, -0.9844872355461121, -0.9940107464790344, -1.3034858703613281, 0.3186769485473633, 1.097107172012329, 0.1690448820590973, -0.4322284162044525, 1.7994909286499023, 0.4739534258842468, -0.8900368809700012, 0.24534212052822113, 1.1701172590255737, -0.9193594455718994, 1.6137433052062988, 1.0898183584213257, 0.21750624477863312, -0.6472241282463074, 1.1784495115280151, -0.7331643104553223, -0.09426773339509964, -0.7347379326820374, 1.8562921285629272, -1.0050384998321533, 0.36228126287460327, 0.43452975153923035, -0.1894582211971283, 0.7522379159927368, 0.16441181302070618, -0.6461576819419861, 1.4854868650436401, -0.3625141978263855, 1.6669764518737793, -1.4691362380981445, -1.503231406211853, 0.3016034662723541, 0.174812451004982, -1.562346339225769, -0.4289642870426178, 0.7558968663215637, -0.19352811574935913, 1.851204752922058, -0.5990481376647949, -0.7435969114303589, -0.5895151495933533, 0.33757755160331726, -0.5572218894958496, 0.851304829120636, 1.0901525020599365, 0.04823718219995499, -0.25282102823257446, 0.7133316397666931, -0.1808352917432785, 0.3699173927307129, -1.687631607055664, -0.09529999643564224, 1.322869062423706, 0.9914628267288208, 1.15571928024292, -0.41546815633773804, -0.5316582322120667, 0.12362372130155563, -0.2169014811515808, 0.7507645487785339, -0.7315531373023987, -0.12095405906438828, 0.8404735326766968, 1.1366868019104004, -1.677160620689392, 1.2236573696136475, -0.15744177997112274, 1.6103870868682861, 0.6664407849311829, 0.8603362441062927, 0.5435298681259155, 1.173041820526123, 0.47682955861091614, 1.490349531173706, -0.29524025321006775, 1.3155885934829712, -1.1906685829162598, 0.7442958354949951, -0.8619896173477173, 0.4769471287727356, 1.421866536140442, -0.4888576865196228, -0.12030429393053055, 0.11042149364948273, -0.5793201327323914, -1.3762166500091553, -0.27189868688583374, 0.45175427198410034, -0.07138002663850784, 1.3576380014419556, -1.596740961074829, 0.820391058921814, -1.2100738286972046, -0.28415918350219727, -0.6500784754753113, 0.482573539018631, 0.5809440016746521, 1.574860692024231, -1.284533143043518, 0.677880585193634, 2.355773687362671, 1.7823983430862427, 1.3992797136306763, 0.5023930072784424, -0.09184665232896805, 1.2336856126785278, 0.21329337358474731, 0.6677765250205994, -0.235778346657753, -0.37034010887145996, -0.9298040866851807, -0.8958950042724609, 0.2994741201400757, 0.47205644845962524, 0.16205115616321564], [-0.37586700916290283, 1.6299207210540771, -0.1985214650630951, 1.8214997053146362, -2.4203333854675293, -0.01347273774445057, 1.7926839590072632, 0.0528029128909111, 0.9621915221214294, -2.04535174369812, -0.24933642148971558, -1.1577924489974976, 0.874201774597168, -0.27862510085105896, -0.0704251378774643, 0.7547042965888977, -1.9298516511917114, 0.576019823551178, -0.07626080513000488, 1.8220423460006714, -0.8620911836624146, 0.07889606058597565, 0.6373481750488281, -0.050430405884981155, -0.3636742830276489, 1.39194917678833, 0.5378996133804321, 0.97589111328125, -0.1691139042377472, -2.624272108078003, -0.9228972792625427, -2.2136969566345215, 0.688292920589447, -1.3162810802459717, 1.0591922998428345, 0.4120687246322632, -0.5247105360031128, -0.5505050420761108, -0.04910087212920189, 1.67936372756958, -0.960141658782959, 0.06021086871623993, -0.6729062795639038, 0.27464786171913147, -1.0052869319915771, -0.05388037860393524, -0.7387291789054871, -1.3762259483337402, -1.5655854940414429, -1.3022449016571045, -1.3339225053787231, 1.4344249963760376, 0.15369448065757751, -0.9665310382843018, -1.1667793989181519, 1.4651204347610474, -1.8579316139221191, -0.07572638988494873, 0.2740146815776825, 0.712262749671936, -0.42181843519210815, -0.43312227725982666, 0.276090532541275, 0.29037541151046753, -0.02462637983262539, -0.1850098967552185, 0.3124468922615051, 0.5520350933074951, -1.0359610319137573, -0.265959233045578, -2.279925584793091, 2.1572823524475098, 0.3303043842315674, -0.00026366228121332824, -0.13228285312652588, -1.2654850482940674, -1.1195058822631836, 0.21094878017902374, -1.0901587009429932, 0.13540560007095337, 1.0178275108337402, 0.5310623049736023, 0.406637966632843, -0.7201770544052124, 0.7915508151054382, -0.7409075498580933, 2.263505458831787, -0.213247612118721, 0.19063933193683624, 0.48053354024887085, 1.3220257759094238, -0.5156602263450623, -0.2187851220369339, -0.7848908305168152, 0.32790717482566833, 0.726261556148529, -0.179646834731102, -1.3303190469741821, -2.598789930343628, -0.43306592106819153, -1.3793925046920776, -1.506048560142517, -0.7339382767677307, -0.42131006717681885, 0.2933862805366516, 1.022145390510559, 0.22592021524906158, -0.14960995316505432, -1.4125590324401855, 1.131988763809204, -1.4684392213821411, 2.6026530265808105, 1.909735083580017, 0.7647992968559265, -0.9584506750106812, -0.8251098990440369, -0.6463921070098877, 0.5478784441947937, 0.31715264916419983, -0.03347140923142433, -0.7026355862617493, -1.3856674432754517, -0.43039026856422424, -0.13353806734085083, -0.03465191647410393, 1.4003069400787354, 1.1847853660583496, -1.0219411849975586, 0.9409368634223938, -1.4001922607421875, 1.27437424659729, 1.249961256980896, -0.6925919651985168, 0.6162182092666626, 1.2608184814453125, -0.7820788621902466, -0.7413644790649414, 0.4300827085971832, 1.058028221130371, 0.6552075147628784, 0.18405267596244812, 0.5661806464195251, 1.9590100049972534, 0.39633023738861084, -0.5590174198150635, -0.5508401393890381, -0.026493502780795097, -0.14694134891033173, 0.8856591582298279, 0.19766512513160706, -0.4845660924911499, 1.2252780199050903, -0.3775891363620758, 1.539935827255249, 0.5394556522369385, -0.8540107011795044, 0.9233476519584656, -0.221345916390419, -2.2290267944335938, -0.5294827818870544, 0.9894962310791016, -0.1011141836643219, 0.9100580811500549, 0.3990672528743744, -1.1758190393447876, -0.21132609248161316, -1.4253133535385132, -1.5033255815505981, -0.12910442054271698, -0.22239966690540314, -1.4045852422714233, 1.1434690952301025, -0.33177506923675537, -0.6026371717453003, 0.487562894821167, -0.9745600819587708, -0.53447425365448, -0.6271051168441772, -0.5545380711555481, 0.7027003765106201, -0.1326555609703064, 1.5424764156341553, 0.8247890472412109, 0.8239733576774597, -0.041952647268772125, -0.3928447663784027, -1.6375460624694824, -0.8045878410339355, 1.4417288303375244, -1.3439723253250122, -0.935014545917511, -0.15052518248558044, 1.1502169370651245, 1.1390459537506104, -1.096924066543579, 0.24033547937870026, -0.7809625267982483, -1.3279610872268677, -0.6750565767288208, 1.672174334526062, -0.4447864890098572, -1.1236292123794556, 0.5626029372215271, -1.7141151428222656, 0.30978089570999146, 0.775816023349762, -1.4216710329055786, 0.6294043064117432, 0.9112333655357361, 1.1865471601486206, 1.1154491901397705, 1.0445369482040405, 0.5735743045806885, -1.5706723928451538, 1.5973045825958252, -0.09443657100200653, -0.8894186615943909, 0.25495895743370056, -1.5307914018630981, 1.1058294773101807, 2.0458309650421143, 1.034108281135559, 0.45285099744796753, 0.6392852663993835, 0.4102942943572998, 0.28534799814224243, 0.658661425113678, 0.0019322384614497423, -1.3100743293762207, 0.9224900007247925, 0.44695404171943665, -1.314131736755371, 0.28925514221191406, 0.899789571762085, -0.36097225546836853, -1.5899393558502197, 0.04283085837960243, 1.1194522380828857, 0.16996373236179352, 1.7480219602584839, 0.00656717037782073, 0.13316039741039276, 0.2258341759443283, 0.5347303152084351, 1.327088475227356, -0.8465880751609802, -0.06558314710855484, 0.2716045379638672, 1.0577329397201538, 1.8586698770523071, 0.5358660221099854, 0.9168403148651123, -0.40210577845573425, -0.37838122248649597, -1.2559165954589844, -2.1346535682678223, -2.811530113220215, 0.6531554460525513, -0.6745506525039673, -0.14655795693397522, -0.18578852713108063, 0.10680023580789566, -0.15161898732185364, -0.6760081052780151, 0.8936045169830322, 1.1340066194534302, 0.023322818800807, 1.3927494287490845, 0.7767539024353027, -0.1984734684228897, -0.21230880916118622, 0.49242305755615234, 0.30224114656448364, -1.9580490589141846, 0.6586971879005432, 0.6570518612861633, 1.532057523727417, -1.9111206531524658, 1.8482297658920288, 0.36778026819229126, -0.3361078202724457, 0.728038489818573, -0.41881853342056274, -0.6156333684921265, 0.002522762631997466, -0.7230054140090942, -1.0630106925964355, -0.4314351975917816, -1.3118796348571777, -0.22393761575222015, -0.1809951663017273, -1.1734451055526733, 0.4757714569568634, 0.6507785320281982, 0.10578034073114395, -2.5752220153808594, -0.43977347016334534, 0.4913640022277832, 0.3743993043899536, 1.8281018733978271, 0.2583252489566803, 1.4799195528030396, 1.8438206911087036, 0.9120458960533142, -0.17395994067192078, -0.5446054935455322, 0.8432595133781433, 1.2202143669128418, 0.12478448450565338, 0.8361302018165588, -1.4889816045761108, 0.44303303956985474, 1.8638991117477417, 0.059464070945978165, -1.4380831718444824, 0.5690759420394897, 1.1829124689102173, 0.5722357034683228, -1.6710903644561768, -0.8098443150520325, 0.14140866696834564, -1.813739538192749, 0.011836194433271885, 0.11479050666093826, -1.0871812105178833, -0.8304899334907532, -0.2672976553440094, 0.3994605839252472, 1.021106481552124, 0.5836480259895325, 2.296774387359619, 0.022994277998805046, 0.6245527863502502, 0.5762252807617188, -0.6851988434791565, 1.1501929759979248, -0.6339066624641418, -0.5337770581245422, -1.3602008819580078, 0.0908476933836937, -0.24788972735404968, 1.1973695755004883, 0.4516349732875824, 0.22493596374988556, -0.1525176614522934, 0.12986353039741516, 0.3027723431587219, 1.313001036643982, -1.1646745204925537, -0.5845323801040649, 0.9908444881439209, -0.4682829678058624, -0.2808413803577423, -1.5698020458221436, 0.365067720413208, -0.9060798287391663, 0.6979015469551086, 0.6876063942909241, -2.1640608310699463, 0.06379541754722595, 1.0631506443023682, 0.7741634249687195, -1.2570528984069824, -2.23704195022583, -0.6214762330055237, 0.6579837203025818, 0.26552629470825195, -0.35374078154563904, -0.21822316944599152, 1.2042168378829956, 1.0309783220291138, 0.3794642984867096, 1.360742211341858, -0.09933800995349884, 1.0825542211532593, 0.8614600896835327, 1.8166978359222412, -0.11546254903078079, 0.6853584051132202, 0.4460490047931671, -0.5987274646759033, -0.7507005929946899, 1.941186785697937, -1.4124845266342163, 1.5322065353393555, -0.01775425486266613, -0.006475906819105148, 0.3591780364513397, -0.39579617977142334, -1.1039763689041138, -2.1552209854125977, 1.7153944969177246, 0.685508131980896, 0.37105756998062134, -0.42892301082611084, -1.0366637706756592, 0.6751763224601746, 1.6117463111877441, 0.3945862054824829, 1.155993103981018, 1.2442270517349243, 0.7424547672271729, 2.3923447132110596, 0.9931067228317261, -1.05986750125885, 0.37417131662368774, 0.6549383401870728, -0.370073139667511, 0.0345463827252388, -0.7440007925033569, -0.7796626687049866, -0.41774532198905945, -0.9598813652992249, 0.45133787393569946, 1.381334662437439, -0.6914215683937073, 0.5340166687965393, -1.7582851648330688, -0.18729232251644135, 0.3361227810382843, 0.4630430340766907, -0.847025990486145, 1.3959720134735107, -0.04429851472377777, -0.37390297651290894, -1.00654137134552, 1.6300338506698608, -0.36566561460494995, -0.08791998028755188, -0.9430491328239441, -1.6894129514694214, 0.4665564298629761, 1.0400478839874268, -0.9536253213882446, -0.05048753693699837, -0.6798278093338013, 0.6618111729621887, -0.3267194926738739, -0.026935787871479988, -0.38196173310279846, 0.041944943368434906, 0.6197763085365295, -0.18967434763908386, 0.8739562630653381, -2.6336328983306885, 2.8122165203094482, -0.9512594938278198, 0.72740238904953, -0.7983822822570801, 0.008161302655935287, -0.10797088593244553, 0.2828047573566437, 1.6834732294082642, -0.9007118344306946, -0.4693340063095093, -0.020702678710222244, -0.22624656558036804, -1.5977933406829834, 0.385206937789917, 0.8705347180366516, -0.07392609864473343, -1.0968471765518188, -1.5718097686767578, 1.3751095533370972, 1.4628530740737915, -0.8769372701644897, -0.6220538020133972, -0.4649834632873535, 0.5447593331336975, 0.2977766990661621, -1.0693563222885132, -1.082572340965271, -0.2097690999507904, -0.5933699011802673, -1.2197645902633667, -1.1233779191970825, 0.39627739787101746, 1.6747400760650635, -1.22046959400177, 0.7311544418334961, 1.3607479333877563, 0.5438672304153442, 0.16927625238895416, 0.5810208320617676, 0.8390210270881653, -0.12138491868972778, -1.5090124607086182, 0.019988274201750755, -0.13420680165290833, -0.08217864483594894, 0.7014439702033997, 0.9644311666488647, -1.5088987350463867, 0.8322640657424927, -2.1467127799987793, 0.7115227580070496, -0.2768445312976837, 0.7507261633872986, 0.4347238540649414, -0.8123974204063416, 1.3965946435928345, 0.8345581293106079, -0.17964346706867218, 0.23364035785198212, 0.11251158267259598, 0.8727702498435974, -0.8351684808731079, 1.623072624206543, 0.5388220548629761, 0.04783593490719795, -0.44270774722099304, 0.1818661242723465, -0.15384900569915771, -0.7065156102180481, -0.4580118954181671, -1.8226454257965088, -0.3519101142883301, -1.4952919483184814, -0.7302751541137695, -1.049338459968567, 2.201845645904541, -0.5166409611701965, 2.342798948287964, -0.6191096305847168, -1.3140993118286133, -0.6190009713172913, 1.4106993675231934, 1.405956506729126, -0.08643190562725067, -0.41316333413124084, -2.484121322631836, -1.1283384561538696, 0.9122769832611084, -0.6777949333190918, -0.42382511496543884, 0.299335777759552, 1.6332651376724243, -0.44473275542259216, -0.09857732057571411, -0.7046791315078735, 1.397773027420044, 0.7964204549789429, 1.4010525941848755, 1.3293019533157349, 1.0300348997116089, -0.006068496033549309, 0.4779075086116791, 3.7925736904144287, -0.9956616759300232, -1.4817352294921875, -0.8148279786109924, -0.6920207142829895, -0.8069037199020386, 0.2399899810552597, -1.501142144203186, 1.3903342485427856, -0.5964163541793823, -1.7375612258911133, 2.3014869689941406, -1.3352560997009277, 1.4196844100952148, -1.3039090633392334, 0.8759057521820068, -0.002786981873214245, -0.29532426595687866, -1.2240395545959473, -0.7959112524986267, 0.8240965008735657, -0.5201126337051392, 0.9505849480628967, -0.3213196396827698, 1.390172004699707, -0.4962054193019867, 1.5092202425003052, 0.3783026933670044, -0.4980311691761017, 0.7979073524475098, 0.2663910686969757, 0.8028801679611206, 0.39209824800491333, 1.3492201566696167, -0.965726375579834, 1.2976480722427368, 1.476996660232544, -1.075461745262146, 1.9284536838531494, 2.0267488956451416, 1.1217451095581055, -0.00043483724584802985, 0.025573614984750748, 2.0466160774230957, -0.3000362813472748, 0.0460796058177948, 1.1260231733322144, 0.7709001302719116, 0.8249501585960388, -0.7787637114524841, 0.7287756204605103, -0.01691354252398014, -0.22210368514060974, -1.227993130683899, -0.5321114659309387, 0.5943740010261536, -1.1218663454055786, 2.6700477600097656, 1.5043227672576904, 0.8981805443763733, 0.20818209648132324, -0.29710811376571655, -0.43818753957748413, -0.9770249724388123, 1.9502780437469482, 0.22589558362960815, -0.5483294129371643, 0.3749745786190033, 0.6636423468589783, 1.2971787452697754, -0.3943866491317749, -1.8701908588409424, 1.7923544645309448, 0.22670333087444305, -1.2515909671783447, -0.42057299613952637, 0.47996076941490173, 0.33874738216400146, -0.42383939027786255, 0.11061618477106094, 0.7077450752258301, -0.27287063002586365, 2.3113651275634766, 1.6885595321655273, -0.8808974027633667, -1.204919457435608, 0.26662716269493103, -0.3939046263694763, -0.953551709651947, 0.5512046217918396, 0.3103847801685333, 1.952807903289795, -0.2248547226190567, -0.9830095767974854, 0.22533392906188965, 0.4583004117012024, 0.272064745426178, -0.7710888385772705, 0.2505986988544464, -0.6236944794654846, -0.10503362119197845, 0.1944350302219391, 0.29594117403030396, 0.7383893132209778, -0.3329213857650757, 0.30471426248550415, 0.5779843330383301, -1.045409917831421, 1.2634172439575195, 0.28909000754356384, -0.1967291533946991, 0.3606072664260864, 0.31852227449417114, 0.3606381416320801, -0.19898904860019684, -0.8946536779403687, 0.6104750633239746, 0.07008127123117447, 0.7708066701889038, 0.7336476445198059, 0.07278072834014893, 0.810157299041748, 0.536963164806366, -0.5301136374473572, -0.8117672801017761, 0.43075597286224365, -0.6238117814064026, 1.6745368242263794, -0.5479421615600586, -0.48748016357421875, -0.36903125047683716, -0.6307073831558228, 0.5383553504943848, -0.39545461535453796, -1.5814441442489624, -0.17305243015289307, -0.9826499819755554, -0.8397888541221619, 0.5948542356491089, 0.17746011912822723, 0.1524851769208908, 1.2702733278274536, 0.4244604706764221, 0.7560918927192688, 1.5661022663116455, 1.269874930381775, -1.4540314674377441, 0.3355081379413605, -1.1318429708480835, -0.0825788602232933, -0.9522267580032349, -1.045174479484558, -0.41054514050483704, 0.26809221506118774, -0.24441759288311005, -0.10891454666852951, 1.17844820022583, -0.07753192633390427, -0.6324273347854614, 0.24310481548309326, -0.4713110327720642, -0.8563475608825684, 0.03105800226330757, 0.2622450292110443, -0.03166867420077324, -0.9528394937515259, 0.5277381539344788, 0.320084810256958, -0.4607723355293274, -2.3390331268310547, 0.8170779347419739, 1.20979642868042, 0.1370702087879181, -1.026107668876648, -0.779996931552887, -0.07188595831394196, -1.3686275482177734, -1.8524528741836548, -0.35749050974845886, -0.5896142721176147, -0.41759246587753296, 1.6432642936706543, 0.34355902671813965, -1.2747005224227905, 1.3284900188446045, -1.0630464553833008, 0.703317403793335, -1.5415388345718384, -1.0488462448120117, -1.5414355993270874, -1.5654163360595703, -1.0304994583129883, 0.2722206711769104, 0.05001520738005638, 1.248856544494629, 2.4355733394622803, 0.6641461253166199, 0.14067409932613373, 0.6367306113243103, -0.9447109699249268, -1.5629785060882568, 0.44576969742774963, -1.3328858613967896, 0.2533174157142639, -1.4038339853286743, -1.7403695583343506, -0.861974835395813, -0.42248424887657166, -0.1692695915699005, 0.11803403496742249, -0.0026647334452718496, -0.5191212892532349, -0.3997173011302948, -0.12466710060834885, -1.8436291217803955, -1.1984047889709473, -0.2499753087759018, 1.262017011642456, 1.313393473625183, 0.03987566754221916, -0.20418186485767365, -0.8721885085105896, -0.3534298837184906, -1.4210538864135742, 0.8197222948074341, -0.8603162169456482, -0.18320316076278687, 1.0098366737365723, 0.3065866529941559, -0.10089388489723206, 2.098888635635376, -0.2146173119544983, 0.8636876344680786, -0.799957811832428, 0.38269534707069397, 0.5669281482696533, 0.6809331774711609, 0.600447952747345, 1.0358917713165283, 1.161745548248291, 0.25540781021118164, 0.9488750100135803, -0.8261083960533142, -0.008340892381966114, 1.3355315923690796, -0.623393714427948, -1.5787014961242676, -0.37005332112312317, -0.9218825101852417, -0.8610384464263916, 0.08257292956113815, -1.3597291707992554, 0.20505160093307495, 1.482458233833313, -1.1167933940887451, 0.225808247923851, 1.484002947807312, 0.9196118116378784, 0.15064604580402374, -1.4791898727416992, 0.07150581479072571, 0.29459401965141296, -0.6679551601409912, 1.3630874156951904, -0.6789796352386475, -0.7644345164299011, 0.3622399866580963, 1.6692183017730713, -0.4041757881641388, -1.1159124374389648, 0.8140188455581665, 0.2568526864051819, -1.6396504640579224, 1.4131207466125488, 0.8429493308067322, -1.3972216844558716, 0.18909747898578644, 0.4195634424686432, 0.08282492309808731, -0.4956858456134796, -1.643693208694458, 0.5914744138717651, 0.9348520040512085, 1.6893852949142456, -1.3416465520858765, -0.03130471706390381, 0.7006235718727112, -0.08442587405443192, -0.4371231496334076, 0.175786554813385, 0.09894443303346634, 1.6625351905822754, 0.2173808068037033, -0.016270944848656654, 0.9993020296096802, -0.5150561332702637, -0.4828427731990814, 1.3433811664581299, -1.5438965559005737, -1.2888997793197632, 0.6652023792266846, 0.4723847806453705, -0.13639499247074127, -0.6182132363319397, 0.500609815120697, 0.18751344084739685, -0.06511839479207993, -0.4313797950744629, 0.11031249910593033, -0.3520760238170624, -0.6561874151229858, -2.3610706329345703, 0.695199728012085, -0.7557746171951294, -0.18702886998653412, -0.9725050330162048, 0.24535535275936127, -0.7959299683570862, 0.31764915585517883, -0.8773229718208313, -1.0036497116088867, -0.6625454425811768, 1.1055290699005127, -0.416925311088562, -0.6497678160667419, 2.16262149810791, -0.7064475417137146, 1.3693211078643799, 0.7954185605049133, -0.42153263092041016, 0.9579520225524902, -0.6469061374664307, 0.55325847864151, 0.56932532787323, -0.27490365505218506, -0.23360957205295563, 0.7728726267814636, 0.13571320474147797, 0.3815179467201233, 0.019686618819832802, 0.0941547304391861, -0.9623631834983826, -0.29805228114128113, 1.2763975858688354, -0.3532036542892456, -0.8974286913871765, 1.4267827272415161, -1.5412919521331787, 0.862228274345398, 0.03208482265472412, -2.8970744609832764, 0.024074038490653038, -0.7696900963783264, -0.8475220203399658, -0.7176108956336975, -0.6716472506523132, -0.4343111515045166, 0.6278999447822571, 0.29213470220565796, -1.2928624153137207, 0.2816165089607239, -0.33033686876296997, 0.682275652885437, -1.1675145626068115, -0.5537869334220886, 0.7180714011192322, -0.415555864572525, -0.26814770698547363, 0.9165622591972351, -0.6473013162612915, -1.0045138597488403, 0.5505566000938416, 0.14536307752132416, 0.17040255665779114, 0.022362833842635155, 0.10671204328536987, -0.32198336720466614, 1.2832250595092773, 1.3594956398010254, 1.383184552192688, -0.6764323115348816, 1.0669605731964111, 0.6666619777679443, 1.3083977699279785, 0.8435819149017334, -0.8548041582107544, -0.10112500190734863, 0.8660541772842407, -0.94242262840271, -1.3087260723114014, -0.034612640738487244, -0.37801647186279297, -0.11269793659448624, -0.5381137728691101, -0.0752391442656517, -0.5480344891548157, -0.1390010565519333, -0.1486058086156845, 0.20993603765964508, 1.35912024974823, -1.530113697052002, -0.9374670386314392, 0.2621024250984192, -1.675775408744812, 0.318204402923584, 0.7150197625160217, -0.11663690954446793, -0.01820988766849041, -0.3640609085559845, 0.43613094091415405, 0.593575656414032, -0.01917988806962967, -0.8028087019920349, -0.06866096705198288, 1.4541157484054565, 0.16158336400985718, -1.6326669454574585, 1.0554770231246948, 1.9749233722686768, 0.0764661431312561, 0.5000375509262085, -1.0933494567871094, -0.749895453453064, -0.6479067206382751, -0.21290317177772522, 0.9948093295097351, 0.155676007270813, -0.07308148592710495, -0.544954240322113, 0.3735119104385376, 1.8056033849716187, -0.5149528980255127, -0.955973207950592, -3.3034305572509766, -0.013782380148768425, -1.7391525506973267, 1.2170079946517944, 0.5062721371650696, 0.8152384161949158, 0.16131597757339478, 0.3636012077331543, 0.5986024737358093, 0.9572072625160217, -1.8756687641143799, 0.7067083716392517, 0.08108885586261749, 0.21396411955356598, 0.5328959822654724, 1.9319690465927124, 1.0953221321105957, -0.6366062760353088, -1.5651061534881592, -0.8988069295883179, 0.2833369970321655, 0.025044305250048637, -0.15968675911426544, -1.7440600395202637, -0.5252100229263306, -0.34993603825569153, 0.004944930784404278, 0.2617511749267578, -1.9435311555862427, -0.8952927589416504, 0.21907956898212433, 1.2501232624053955, -0.7973929643630981, -0.5378365516662598, -0.32369768619537354, -0.42254650592803955, 0.44785046577453613, 0.07738453894853592, -2.2422356605529785, 0.997624397277832, 0.6990417242050171, -0.9501525163650513, -0.22681786119937897, -0.2454615831375122], [-0.41132816672325134, 1.3776353597640991, 0.011937350034713745, -1.4142756462097168, -1.7341532707214355, -0.20401465892791748, 0.31201136112213135, -1.130996584892273, 1.3754891157150269, -0.11702200770378113, 0.5298682451248169, -0.42537495493888855, -0.7354496717453003, -0.11656723916530609, -0.7884142398834229, 0.7004420161247253, 0.06419414281845093, 0.07549279183149338, -1.5854262113571167, -0.05057842656970024, -1.1556185483932495, -1.0101325511932373, -1.8163655996322632, 0.03210720047354698, 0.190888449549675, -0.7948300838470459, 0.5306618213653564, 1.4722286462783813, -0.012134670279920101, 0.1084042638540268, 1.2913305759429932, 0.44313910603523254, 0.5597507357597351, -0.3655923902988434, 0.5045214891433716, -1.108668565750122, -0.9392593502998352, -1.0560272932052612, 0.3845241665840149, 0.16317246854305267, 0.6904995441436768, -1.4911447763442993, 0.39803600311279297, 0.7077690362930298, 1.4883790016174316, 0.033403776586055756, 1.2809985876083374, -2.6184945106506348, 1.454203486442566, 1.0062897205352783, 0.743134617805481, -0.510296642780304, -2.1814041137695312, 0.5304023623466492, -0.4918522834777832, -0.5111770629882812, -1.0793782472610474, 0.9153128266334534, -1.9643383026123047, 0.9449470043182373, -0.814384400844574, 1.1251142024993896, -0.0724371075630188, 0.7886334657669067, 1.3852880001068115, 0.30814048647880554, 2.3171255588531494, -0.5044582486152649, 1.724469780921936, -0.43250247836112976, -0.98807293176651, 0.28346896171569824, 1.553932785987854, 1.6683202981948853, -0.0068068052642047405, 0.8112475275993347, -1.8040382862091064, 1.647297739982605, 0.19715037941932678, 2.3333587646484375, 0.7520263195037842, -0.2587568759918213, -0.7219051122665405, -0.6465879678726196, -1.412979245185852, -0.8140608072280884, 1.3968819379806519, 0.6749695539474487, -0.0380876399576664, 0.724683940410614, 0.7805012464523315, -1.0179064273834229, -0.21010130643844604, -0.011966660618782043, -1.0322368144989014, -0.349048376083374, -0.41763433814048767, 1.0475564002990723, -0.5801355838775635, 1.0123844146728516, -0.023121962323784828, -0.13330422341823578, -0.5250557661056519, -1.2873142957687378, -0.023107804358005524, 1.6185864210128784, -0.969199001789093, -0.9688055515289307, 0.2766243815422058, 1.7779161930084229, -0.0756136029958725, -0.14736111462116241, 1.67249596118927, -0.7710989713668823, 0.09154531359672546, 0.17045798897743225, 0.4531205892562866, -0.2014314979314804, -0.8064115047454834, -1.766374111175537, -1.406461238861084, 1.5233207941055298, -2.330822229385376, 1.3972375392913818, -0.9785351753234863, -2.032214403152466, -0.007223017048090696, 0.3579319417476654, 0.1895199418067932, 0.11864869296550751, 0.5285951495170593, -1.102803111076355, 0.15262606739997864, 0.5376157760620117, -0.7623604536056519, 0.38182738423347473, 0.10975377261638641, -0.5418155789375305, -0.9461628198623657, -0.45353373885154724, 0.8010349273681641, -0.1746400147676468, -1.0368046760559082, -1.0799113512039185, -0.41942888498306274, -0.023228393867611885, -0.4490680396556854, 0.15529681742191315, -1.1211868524551392, -0.8482467532157898, -0.8078335523605347, 2.112926721572876, -0.45670217275619507, -0.0566023513674736, 0.20151907205581665, -0.1870880424976349, -0.8112392425537109, 1.3814079761505127, -0.14362525939941406, -0.1474636197090149, 0.32926392555236816, 0.20586229860782623, 0.1131986454129219, -0.17716893553733826, -0.5980344414710999, -1.5497939586639404, 0.4579790532588959, 0.5951794981956482, -0.6706213355064392, 0.8538174629211426, 0.6049636602401733, -1.833476185798645, 0.41334185004234314, 0.18296682834625244, -0.35993295907974243, 0.34486743807792664, -0.2282080203294754, -0.4029944837093353, 0.38798803091049194, 1.1849919557571411, -1.4018168449401855, -0.46542811393737793, -1.0884544849395752, -0.732774555683136, 0.20371824502944946, -0.8468362092971802, 0.31003260612487793, 2.560579776763916, -0.36878740787506104, -0.3257921040058136, -0.5651082992553711, -1.4065113067626953, 1.3632452487945557, -0.5738458037376404, -0.41744980216026306, 0.7269452214241028, -1.793413519859314, 0.5366107225418091, 0.6635519862174988, 0.6339481472969055, 1.454775094985962, -0.36834654211997986, -0.11258574575185776, -0.4162609875202179, 0.5089700818061829, 1.6173161268234253, -1.0185418128967285, 1.5354640483856201, 1.352612853050232, -1.394344687461853, 1.299114465713501, -2.0479907989501953, -1.554916501045227, -1.270808458328247, 2.888824939727783, -1.2262892723083496, 1.0762133598327637, 0.31635308265686035, -1.05924391746521, 0.2225155234336853, 0.1333046555519104, 0.031358662992715836, -0.7608773708343506, 0.7958978414535522, 2.0993692874908447, 0.17698204517364502, 0.15671345591545105, 2.5312511920928955, 1.591673731803894, -1.2419604063034058, -1.7685654163360596, 0.8051790595054626, -1.1288715600967407, 0.9840848445892334, -1.305700421333313, 1.6286845207214355, -0.36079713702201843, -0.1167980283498764, -1.165173888206482, -0.4285135269165039, 0.015303145162761211, -0.607654333114624, -0.22058603167533875, -1.9682667255401611, -0.9985857009887695, -1.949869990348816, 0.30976739525794983, 0.7994105219841003, 0.39659157395362854, 0.9222338199615479, 0.3904130458831787, 0.20579206943511963, 0.4190237820148468, -1.3338152170181274, -0.9851004481315613, 0.8652149438858032, 0.07098495215177536, 0.8073068261146545, 0.19316606223583221, 0.801393985748291, -1.3141177892684937, -1.9034225940704346, 0.046063825488090515, -0.23425067961215973, 3.2666687965393066, -0.6025813817977905, 0.15815436840057373, -0.21681959927082062, 0.4904535114765167, -0.8919922113418579, 0.5522196292877197, 2.3995513916015625, -0.31851497292518616, -1.2845747470855713, 1.1057347059249878, 0.33686569333076477, -2.114224910736084, 1.0870004892349243, -0.7035925388336182, -0.0620003342628479, -0.4440905451774597, 0.5084933042526245, -1.1592166423797607, 0.24411675333976746, -1.032224178314209, -2.104435920715332, 0.17089861631393433, 0.003592671127989888, 1.9453136920928955, 0.5977659225463867, 0.6765653491020203, 0.5585005283355713, 1.8198848962783813, -0.9010365009307861, -0.7195343971252441, 0.39917173981666565, -0.5125076174736023, 0.61263108253479, 0.8931689858436584, 0.33023539185523987, 0.596621572971344, 1.2366360425949097, -0.6845749020576477, 0.048891905695199966, -0.798032283782959, 0.5289916396141052, 1.0723092555999756, 0.5384241342544556, -0.20272208750247955, -0.7718080282211304, -0.6151758432388306, 1.9902007579803467, 0.8248990774154663, 0.10432720929384232, 0.013451133854687214, -0.055093005299568176, 0.051472198218107224, -0.15195640921592712, -1.6633093357086182, -0.6190160512924194, 0.4735664129257202, -1.9954286813735962, 1.4221656322479248, -0.008286140859127045, -0.6390715837478638, -0.7709373831748962, -2.3947994709014893, 0.20593199133872986, -0.23453716933727264, -0.2781834900379181, -0.9758751392364502, 0.3257876932621002, -0.3056235611438751, -1.0354905128479004, 0.9866126179695129, -1.1394790410995483, 0.37449929118156433, -0.2979837656021118, -0.49612659215927124, -0.2948914170265198, 0.8344114422798157, -1.0701185464859009, -0.2926468253135681, -1.3688963651657104, -0.17598339915275574, 0.2755872309207916, -0.2959485352039337, 1.0921896696090698, 1.5413849353790283, -0.18348921835422516, -0.6864319443702698, -0.6778463125228882, -0.8041150569915771, -0.6379081606864929, 0.2435263842344284, 0.4670615494251251, 0.1568586230278015, -0.1993693858385086, 0.2589868903160095, -2.1054792404174805, -0.8506028652191162, 0.1600053757429123, -0.43470999598503113, 0.3699755072593689, 0.06573119014501572, 1.6182608604431152, 0.6134637594223022, -2.3551712036132812, -1.3150042295455933, -0.30474910140037537, -0.978398323059082, 0.723017156124115, -0.4988589882850647, 0.3026005029678345, 1.4182847738265991, 0.30748894810676575, 0.7912349700927734, 1.3118135929107666, 0.3642938435077667, 0.45408034324645996, -2.052138328552246, -0.24559107422828674, -0.7740626335144043, 0.7866197824478149, -0.3048867881298065, 1.2675966024398804, 2.2095468044281006, -0.33527135848999023, 0.6264859437942505, 0.2807183265686035, 0.24972230195999146, -0.07837342470884323, 0.6328063607215881, 0.14320644736289978, 0.44909587502479553, -1.2548882961273193, 0.3858349919319153, -0.44073137640953064, 0.22193501889705658, -3.310391426086426, -1.2561432123184204, -1.6622157096862793, 1.0523769855499268, -0.42447999119758606, 0.714760959148407, 0.20410913228988647, -0.04442601650953293, 0.3294599652290344, -0.17549175024032593, 0.20133076608181, 0.736780047416687, -2.365051507949829, 1.8671786785125732, -0.2695487141609192, -0.5374423861503601, -0.1736040711402893, 0.2090393602848053, 1.143757700920105, 1.9139480590820312, 0.5818383693695068, 1.4571086168289185, 0.19161207973957062, 1.292064905166626, 0.18234317004680634, -0.5083531737327576, -0.15761803090572357, 0.7617064118385315, -0.6301143169403076, 0.5851301550865173, -1.3755751848220825, 0.47190284729003906, 0.3543040156364441, -0.7571718096733093, -0.44050970673561096, 0.9409800171852112, 0.2251882702112198, 0.06249034032225609, 1.1574442386627197, -0.172211155295372, -1.0928548574447632, -0.25224265456199646, 0.9104576706886292, 1.7450084686279297, -0.9782021045684814, 0.5576330423355103, -1.7853528261184692, -1.0637321472167969, -0.3842681646347046, -0.6496118903160095, -0.5894969701766968, -0.1679944396018982, -1.7757351398468018, -0.9307438135147095, 0.8810207843780518, 0.957190215587616, 0.11400240659713745, -0.9084459543228149, 1.3901885747909546, 0.5476605892181396, 0.803270161151886, 0.09129700064659119, 0.5376062989234924, 0.5029133558273315, -0.05389486998319626, 0.04471433907747269, -1.7157037258148193, 0.8078166246414185, -0.4896903336048126, -0.4769634008407593, -2.1481027603149414, 1.2304388284683228, -0.8392232060432434, -0.5872645974159241, -1.6952906847000122, 0.05980060622096062, -1.6906309127807617, -0.6698121428489685, -0.8640874028205872, -0.3260907828807831, 2.4179229736328125, -0.5084242820739746, -0.7305685877799988, -0.4591970145702362, -0.13615445792675018, -0.019901910796761513, 2.0401644706726074, -0.27968454360961914, -0.051996804773807526, 1.1508580446243286, -1.7341240644454956, 0.2521968185901642, -0.6467082500457764, 0.5339767932891846, -0.3725324869155884, 0.565240204334259, -1.6944892406463623, 0.18923351168632507, -1.0768272876739502, 0.028253041207790375, -0.15747326612472534, -0.4307738244533539, -0.4858156740665436, 0.5002825856208801, 2.0191431045532227, -0.2665814161300659, 0.05366556718945503, 0.22101084887981415, 0.6222572922706604, 1.1556555032730103, -0.49578526616096497, -1.1512722969055176, 2.55881404876709, 0.5865113735198975, 0.19379109144210815, 1.8498255014419556, -1.80046808719635, -0.6514862179756165, 0.5651693344116211, -0.10236495733261108, 1.100570559501648, -0.5876487493515015, -0.5029935240745544, -0.708965539932251, -0.9421425461769104, 0.3382773995399475, 0.572050929069519, 0.32286059856414795, 0.09663918614387512, 0.8856431841850281, 0.7942283749580383, 0.21218723058700562, -0.3046126365661621, 0.2089279145002365, -1.349965214729309, -0.22396327555179596, -0.3626755177974701, -0.5434362888336182, 0.4583229422569275, 3.0942091941833496, 0.06930891424417496, -0.4063689708709717, 2.3487727642059326, -0.3082350492477417, 0.626047670841217, 0.13657855987548828, -0.36485278606414795, 0.7924607396125793, -0.12062173336744308, 0.269087553024292, 0.9216350317001343, -0.8846218585968018, -0.694767951965332, -1.1316930055618286, 0.6982501149177551, -0.03548980504274368, 1.6919406652450562, 0.1463540941476822, -0.32759976387023926, 0.09527338296175003, 1.9691083431243896, 1.2086423635482788, 0.9509605765342712, 0.12159628421068192, -0.26490214467048645, 0.14903436601161957, 0.5876579284667969, 0.34057918190956116, 0.05902917683124542, -0.21108588576316833, -1.0269478559494019, 1.5117696523666382, 1.007803201675415, -0.9194986820220947, -0.5729586482048035, 0.08155155926942825, 0.8758478760719299, -1.4261318445205688, 0.3383798897266388, 0.6796140074729919, 1.311547040939331, 0.8751054406166077, 0.4783235192298889, -0.10905605554580688, -0.3448084592819214, 0.4247090816497803, 0.2753964066505432, 1.5489274263381958, -0.46197038888931274, -1.1012489795684814, -1.5383862257003784, 0.4274962544441223, 0.3604056239128113, -0.631698727607727, 0.6242206692695618, -0.9004412889480591, 0.18114611506462097, -0.588941752910614, -0.5806905031204224, -1.5811209678649902, 0.5696505904197693, 0.5397720336914062, 0.7464749217033386, 1.564541220664978, 0.014990823343396187, 0.35194075107574463, -0.12182146310806274, 0.4244060814380646, -3.145770311355591, -0.3422231674194336, -0.35258209705352783, -0.20258402824401855, 1.6869827508926392, 0.4851069450378418, -1.3489524126052856, -0.6869785785675049, -1.0497441291809082, 0.4132453501224518, 2.0247769355773926, 0.20606869459152222, 0.8973300457000732, -0.0853378102183342, -2.214249849319458, 0.4171585738658905, -0.10939362645149231, 0.048168979585170746, -0.4102618992328644, -0.9126478433609009, -1.8441593647003174, -0.5925523042678833, -0.07982634752988815, 0.485673189163208, -0.7390319108963013, -0.44721201062202454, 0.6254796981811523, -0.760819673538208, -0.42289838194847107, 2.216611862182617, -1.343510389328003, -0.45589926838874817, 0.10069837421178818, -0.15230900049209595, -1.6443920135498047, -0.44952869415283203, -0.39011767506599426, -0.025385992601513863, 1.6750226020812988, 1.908217430114746, 0.2974921464920044, 2.7820606231689453, 1.1848191022872925, 0.13291801512241364, 0.7240382432937622, 0.6889527440071106, 0.21049588918685913, -0.2966254949569702, -0.13929317891597748, -0.7088496088981628, 0.056175991892814636, -0.848734974861145, 0.16008858382701874, 0.2469550222158432, -0.7879942059516907, -0.8216179609298706, 0.117622509598732, -0.07316158711910248, -0.8484557867050171, -0.6860749125480652, -0.005464453250169754, 0.9530375003814697, -0.8927537798881531, -0.6568120718002319, -0.23889414966106415, 0.24053312838077545, 0.3618791997432709, -1.7544726133346558, -0.2731809616088867, 0.5265164971351624, 1.0964434146881104, 0.19120633602142334, -0.8647658824920654, -0.7976179718971252, -0.36763331294059753, 0.021143116056919098, 0.2323080599308014, 0.4131167531013489, 2.1933867931365967, 1.2816145420074463, 0.27569839358329773, 0.10899003595113754, -0.6653167009353638, -1.8255759477615356, 0.31446099281311035, -1.893764615058899, 1.8634178638458252, 0.5062381029129028, -0.26543372869491577, -0.6527777314186096, 0.6238499283790588, 0.19554756581783295, -0.6827852129936218, -0.4489857256412506, -1.4802496433258057, 1.241376519203186, 0.23520004749298096, 1.2085034847259521, 0.42968010902404785, 0.5777854919433594, 1.4312832355499268, -0.41120508313179016, -0.9503586888313293, -1.3967090845108032, -1.6462610960006714, -0.21673372387886047, 1.1086405515670776, -1.4732637405395508, 0.0008174589020200074, -2.046306610107422, 0.11839623004198074, 1.391778826713562, 0.5863777995109558, 1.1774874925613403, 0.8707237839698792, -0.06566543877124786, -0.8107123970985413, 2.3239028453826904, 1.1791471242904663, 0.5322244763374329, 0.23888881504535675, 1.7213099002838135, -1.282874584197998, 0.5328361988067627, 1.211348295211792, 0.6166338324546814, 1.414217472076416, 2.428955316543579, 0.09848908334970474, 0.1367092728614807, -0.9182072281837463, 1.3016670942306519, 0.32088327407836914, -0.46863001585006714, 2.063136339187622, -0.7666000723838806, 0.6088236570358276, -0.9183040857315063, -0.42583584785461426, -0.7801427841186523, -0.17420829832553864, -0.7085093855857849, -0.8262370228767395, 1.1218321323394775, 2.2596302032470703, -1.0607167482376099, 0.48294511437416077, -0.44698768854141235, -0.7192365527153015, 0.8518296480178833, 0.2800242304801941, 0.12835395336151123, 0.3809840977191925, -0.9008030295372009, 1.3243744373321533, 1.0568081140518188, 0.11971783638000488, 0.6518577337265015, -0.7810871005058289, 0.27239152789115906, -0.2518996298313141, -0.0874214768409729, 0.32798925042152405, 1.7921398878097534, 0.6688928604125977, 0.5961722135543823, 0.25397831201553345, 1.4089144468307495, 1.0290610790252686, -0.437786728143692, -0.665044367313385, -0.21523365378379822, -0.9878483414649963, -0.681269109249115, 0.43050840497016907, -1.2413698434829712, -0.3997485935688019, 0.887524425983429, 1.0110454559326172, -0.808070719242096, -0.21254025399684906, 0.13794024288654327, -1.1281095743179321, -1.166103720664978, 0.4171045124530792, 1.212693691253662, -0.3445813059806824, 0.41177815198898315, 0.951678454875946, -1.8999481201171875, -0.1416965126991272, 1.8204870223999023, 0.7705677151679993, -0.36956360936164856, -0.5878468751907349, -0.23297666013240814, 0.28347450494766235, -0.9760057330131531, 1.5335694551467896, 0.21321874856948853, 1.1950278282165527, 1.4800547361373901, 0.6746059656143188, 0.365584135055542, 0.2734384834766388, -0.5466664433479309, -0.025312639772892, -0.6591139435768127, 0.5241562128067017, -0.44311410188674927, -0.29567140340805054, -0.21643167734146118, -0.24596843123435974, -0.6583676338195801, -0.3284843862056732, 0.48928648233413696, 0.07090127468109131, 1.4934083223342896, 1.1871446371078491, -0.6321229934692383, -0.9537643194198608, -0.8640614748001099, -1.017278790473938, 1.7588894367218018, -0.4031887650489807, 1.9592626094818115, 0.05639558285474777, -0.6067314743995667, 0.9892860054969788, -1.3249784708023071, 1.0877071619033813, 1.153324007987976, -0.6579597592353821, 0.06530707329511642, -1.359417200088501, 0.17403605580329895, 0.7453848719596863, -0.4850470721721649, 1.0245096683502197, 1.0208749771118164, -0.656624436378479, -0.08891778439283371, 3.186969041824341, -1.0473867654800415, -0.3115485906600952, -0.11692479252815247, -0.14581869542598724, -2.0171916484832764, -2.345940589904785, -1.175596833229065, -1.9652296304702759, -0.8267316222190857, 1.360576868057251, 0.4598759114742279, 0.12175708264112473, 0.6581351161003113, 1.2186570167541504, 0.4770716726779938, -0.38888126611709595, -0.346310019493103, 0.6013204455375671, 0.8135147094726562, -0.8675779104232788, 2.0494353771209717, 1.3471336364746094, -1.7738056182861328, -1.7563549280166626, -0.8744412660598755, -1.4660030603408813, 0.7177215814590454, 0.7559269070625305, 0.21911948919296265, -0.4418034255504608, -0.5679494142532349, -0.05051658675074577, -0.5197376012802124, -0.6208785176277161, -0.7636988162994385, -0.12107864767313004, 0.06625667214393616, 0.09441390633583069, -0.5574467778205872, 0.9097718000411987, 0.24762316048145294, -0.33978503942489624, 1.2564905881881714, 0.8777992129325867, -2.021923065185547, -0.828541100025177, -0.9645304083824158, 0.502700924873352, -0.6016226410865784, 1.547461986541748, 0.2619651257991791, -1.2548742294311523, -1.0705918073654175, 0.06535688042640686, 0.5173097848892212, -0.1902613639831543, -0.13885380327701569, 0.41814133524894714, -1.6320233345031738, -0.37915000319480896, -1.3661961555480957, -0.07485480606555939, 0.013693689368665218, -0.24383263289928436, 0.4908277690410614, -0.020704466849565506, 0.1206740215420723, 0.36566776037216187, -0.17742806673049927, 1.086491584777832, 0.165889710187912, -1.3204187154769897, 2.4284305572509766, -0.5299777984619141, 0.3137878477573395, 0.7703897356987, -0.4732396900653839, 0.6302646994590759, -0.7306691408157349, 0.0263169277459383, -0.09971123188734055, -0.7806169390678406, 0.9196392893791199, -1.8657454252243042, 0.7169864177703857, 0.405741810798645, 0.707952082157135, -0.9000983834266663, 2.03583025932312, 0.5536832213401794, -0.38764920830726624, 1.571426272392273, 2.1202774047851562, 1.0628913640975952, 0.6983663439750671, 0.36708998680114746, 1.1288667917251587, 0.9633554816246033, -0.21982549130916595, 0.4665478765964508, 1.74779212474823, -0.5785701870918274, 1.5845805406570435, -1.2054167985916138, -0.15033574402332306, 0.181551992893219, -0.07811438292264938, -2.272273540496826, -2.3286356925964355, -0.4305254817008972, 2.384665012359619, -1.292047142982483, -1.3865351676940918, 0.7366603016853333, 0.06630109250545502, 2.6844723224639893, -1.0985610485076904, -0.9332559704780579, -0.9024767279624939, -0.30545294284820557, 0.47083351016044617, 0.9643295407295227, 0.3486024737358093, 0.5064879655838013, 0.5881913900375366, 0.7660802602767944, 0.46074196696281433, -0.37019655108451843, -0.3641129434108734, 1.1492204666137695, -0.6332165002822876, -3.254065990447998, -1.1959152221679688, 0.8299415111541748, 1.8630187511444092, -2.990185022354126, 1.1944379806518555, -0.2895379662513733, -0.4968889653682709, -1.8810303211212158, -1.6664506196975708, 2.9001097679138184, 0.17026028037071228, -1.0684959888458252, 0.7970773577690125, 1.0028713941574097, 1.593103289604187, -0.742017388343811, 1.7683767080307007, -1.3629939556121826, -0.9734303951263428, 0.34789127111434937, -0.276021271944046, 0.12443581223487854, 0.4123578369617462, 0.1380188763141632, 0.09915859252214432, 0.6220924854278564, 1.3884249925613403, -0.7682953476905823, -0.4947476089000702, 1.3568651676177979, 0.9893280863761902, 0.37713780999183655, -0.9346040487289429, 1.1072170734405518, 0.6594956517219543, -0.8029367923736572, 0.2525189220905304, -0.599265456199646, -2.522862434387207, -1.2805818319320679, -0.8354891538619995, -0.8220095038414001], [-1.2656581401824951, 0.5275740027427673, -0.7754526734352112, 0.8836048245429993, 0.7143767476081848, 0.13638243079185486, -0.1734451800584793, 1.4218381643295288, -0.8405874967575073, 0.4999827742576599, -0.8874956965446472, 0.19937020540237427, -1.6599633693695068, -1.581092119216919, -0.5042562484741211, -0.5154700875282288, -0.11277883499860764, -0.9007993936538696, 0.8574047088623047, -0.3448465168476105, -0.5895537734031677, 0.15341922640800476, -2.95273494720459, 1.288211464881897, 1.1736265420913696, -0.869571328163147, -1.8177273273468018, -1.8611321449279785, -0.4121856689453125, 0.07923898100852966, -0.8994308114051819, -1.2342334985733032, -0.5376483201980591, 0.9282370209693909, -1.1174850463867188, 1.0075037479400635, 0.04149096459150314, 0.5395927429199219, 1.2452397346496582, -0.4033057391643524, 0.8267269134521484, 0.2909751534461975, -1.6304672956466675, 0.29895928502082825, -0.9808683395385742, 0.0064588249661028385, -0.2686312794685364, -0.012115245684981346, 0.3520164489746094, 0.664220929145813, -1.2035632133483887, -0.4539491832256317, -1.2264448404312134, 0.9472060203552246, -0.16364173591136932, -0.21198904514312744, -0.7787838578224182, 1.0148026943206787, 0.47643280029296875, 2.8301985263824463, -0.6917499899864197, 0.46110907196998596, 0.3070966601371765, 0.35165315866470337, -0.6843012571334839, 0.8946397304534912, -0.6866343021392822, 0.7515808939933777, 0.18408797681331635, -0.8163175582885742, -0.5199813842773438, -0.3867439031600952, -0.6118103861808777, 0.3055287301540375, -0.24668873846530914, 0.4533146619796753, 0.28306227922439575, -1.2748743295669556, -0.28483378887176514, 0.8029419183731079, 0.5104251503944397, -1.3685718774795532, 0.8756966590881348, 0.3484550714492798, 0.40870076417922974, -0.200674906373024, -0.7600222826004028, 1.3038638830184937, 0.17206959426403046, -0.31075477600097656, 1.1814473867416382, 1.1965636014938354, 0.9315634369850159, -0.0535782054066658, -0.5296031832695007, -0.8987072706222534, -0.07026743143796921, -0.45387235283851624, 0.6925382018089294, -1.1734627485275269, 1.4150632619857788, 0.1668897122144699, -0.6222695112228394, -0.49147719144821167, -0.041690751910209656, 1.734720230102539, -1.5887519121170044, 1.418243408203125, 0.9061551690101624, -1.2825617790222168, 2.57070255279541, 0.32923024892807007, 0.21317753195762634, -0.4402756094932556, -0.9815059900283813, -0.014082377776503563, 0.39174121618270874, -0.24601568281650543, 2.628898859024048, 1.1843819618225098, 1.631166696548462, 0.5685971975326538, 0.3840874135494232, -1.0883774757385254, 1.4927188158035278, -0.39419230818748474, 1.9816633462905884, 1.7830814123153687, 1.4622223377227783, 1.3292890787124634, -0.4167015850543976, -0.5808359384536743, -0.5703248381614685, -0.5406312346458435, 1.9851534366607666, 3.501600742340088, 0.8067130446434021, 0.8642627596855164, 0.058454643934965134, -0.7801638245582581, -0.6924998164176941, 1.0116580724716187, 0.09306640177965164, 0.729739785194397, -0.3214734196662903, -0.6322900056838989, 0.4136582314968109, 0.34447529911994934, -0.19801340997219086, -0.7391017079353333, 1.1397273540496826, -0.3953527510166168, -1.0892107486724854, -0.0009615370654501021, -0.5468276143074036, -0.014159184880554676, 0.8700195550918579, 0.08889292180538177, -1.0778348445892334, -0.39349111914634705, 0.27585336565971375, -1.1062887907028198, 0.994837760925293, -0.36040616035461426, 1.0815978050231934, -1.0036653280258179, -1.4755662679672241, -0.9507718682289124, -0.8611252307891846, -0.13381443917751312, 1.3119815587997437, -0.08079206943511963, 0.28850677609443665, 0.0877930149435997, -0.2395988404750824, 0.5644436478614807, 1.2653995752334595, -1.5962169170379639, 1.0821837186813354, 0.28831174969673157, -0.788410484790802, 0.9440273642539978, 0.026254279538989067, -0.5687068700790405, 0.03296274319291115, 0.24201640486717224, 1.2224342823028564, 0.5638862252235413, -0.34630706906318665, 0.29640722274780273, 0.897795557975769, 1.1827665567398071, 0.3390487730503082, -0.9106353521347046, 0.09779899567365646, -0.5224021077156067, 1.6464860439300537, -0.31733113527297974, -1.2025539875030518, 0.2735879719257355, -0.7655317187309265, -1.1306513547897339, 0.22139349579811096, -1.6997171640396118, 0.8084862232208252, 2.411939859390259, -0.15840941667556763, 0.016895586624741554, 0.41691359877586365, -0.9342067837715149, 0.7641630172729492, 0.2416009157896042, -0.6454423069953918, -0.4671151041984558, 0.4720260500907898, 0.21645520627498627, -0.9101431965827942, 0.2297484278678894, 0.24341243505477905, 0.7641193270683289, 1.551518201828003, -0.164608433842659, 0.49971142411231995, 0.11901538819074631, 0.11595851927995682, 1.4321062564849854, 0.11390568315982819, 0.39940449595451355, -0.03712887316942215, 1.384061336517334, 0.21410110592842102, 0.017767025157809258, 1.1407568454742432, -0.2650037407875061, 0.7197628617286682, 0.5858738422393799, -0.3341985046863556, -1.6921193599700928, -0.2894414961338043, 0.6442826390266418, 1.0176225900650024, 0.6385661363601685, 0.666977047920227, -0.22226622700691223, 0.23296204209327698, -1.915715217590332, -0.6807764768600464, 1.0674691200256348, -0.16103197634220123, 1.4896562099456787, -1.4576396942138672, -0.6532981395721436, 0.8002954721450806, 0.924669086933136, 0.442727655172348, 0.058214638382196426, -0.41065317392349243, 1.4409818649291992, 0.2728649079799652, -0.4617656469345093, 0.14983704686164856, -0.1440032720565796, 1.1398191452026367, 0.09681056439876556, -0.6489852666854858, -0.889719545841217, -0.6177637577056885, -1.4558871984481812, 0.2637891173362732, -0.8076409101486206, 1.424803614616394, -1.2223764657974243, -0.678847074508667, -1.5450743436813354, -1.3764898777008057, 0.2429608404636383, -0.32288622856140137, -0.8854985237121582, 0.18163470923900604, 0.1694926917552948, -1.0460695028305054, -0.7148546576499939, -0.4013580083847046, 0.5896630883216858, 0.3427397608757019, 1.156241774559021, -0.3637189567089081, -0.619019091129303, -0.7257084250450134, -0.21272797882556915, 0.3919360637664795, -1.267960548400879, 0.9101308584213257, -0.05186757072806358, 0.5952311754226685, -1.3368277549743652, -0.34797266125679016, 0.8003039956092834, 0.060936037451028824, 1.2280634641647339, 1.664819598197937, -0.02199840545654297, -0.7641381025314331, 0.436444491147995, -1.1966257095336914, -0.287602037191391, -0.5197000503540039, 0.7149118781089783, -0.3179697096347809, -0.41495072841644287, 1.4112869501113892, -0.24527235329151154, 0.49803265929222107, -0.44766390323638916, 1.3054125308990479, -0.299216628074646, -1.657725214958191, 1.9391392469406128, -0.18185260891914368, 0.6228787899017334, -1.2743356227874756, 0.01453478541225195, 0.9985631108283997, 1.8581243753433228, 1.7101775407791138, 0.9145157933235168, -1.4801779985427856, 0.48102980852127075, 2.1476588249206543, -1.91982102394104, 0.6684615612030029, 2.3508245944976807, 0.019484905526041985, -1.1266098022460938, 1.3059632778167725, 0.1713227778673172, 1.604483723640442, 2.3093223571777344, 0.9953524470329285, 0.5840572118759155, 0.7310187220573425, -0.1230345219373703, 1.2428550720214844, 0.39579907059669495, 1.1106327772140503, 0.8624181151390076, -0.4786168038845062, 0.2779156565666199, 0.7954810261726379, -1.8063147068023682, -1.0791198015213013, -0.3983173072338104, 1.0363483428955078, -0.45571380853652954, 0.3974519371986389, -0.27193930745124817, 1.9209741353988647, 0.6773480176925659, 0.29972755908966064, 0.7180130481719971, -0.3409220576286316, 0.5138519406318665, -1.701751947402954, 1.2728315591812134, 0.16375413537025452, -0.4574338495731354, 0.18665778636932373, 0.9649807214736938, -1.1526817083358765, 0.024280181154608727, 0.5227159857749939, -1.158874273300171, -1.4785666465759277, 0.2717961370944977, 1.5799782276153564, -0.18048213422298431, 1.8025585412979126, 1.310651183128357, 1.122128963470459, -1.7643251419067383, -0.0008447542786598206, 0.24274523556232452, 2.105962038040161, -0.2503000497817993, -0.18045158684253693, -1.1701343059539795, -0.03368194401264191, 1.939134955406189, -1.8611388206481934, -2.841797113418579, 0.054561976343393326, -2.632031202316284, -1.5514296293258667, -0.11620928347110748, -0.5521537065505981, 0.7288128137588501, 0.5013974905014038, 1.358127236366272, 0.0594157837331295, 0.28642719984054565, -1.295046329498291, 0.2676566541194916, 0.9956474304199219, 1.0461293458938599, -1.130424976348877, 1.161554217338562, -0.630189836025238, 0.41423970460891724, 1.2343709468841553, 0.06523852050304413, -0.010040875524282455, -0.024275751784443855, -1.619803547859192, -0.2204892784357071, 0.6519584655761719, 0.15561005473136902, -1.6870191097259521, -0.8593527674674988, 0.4824150800704956, 0.06665538996458054, -1.071944236755371, 0.7482624650001526, -2.4678075313568115, 0.9504087567329407, 1.7590793371200562, 0.2635405659675598, 0.45193490386009216, 0.1614091545343399, 0.19989155232906342, 2.3385772705078125, -0.10861359536647797, 0.523958683013916, -1.425338864326477, -0.8622093796730042, 1.4252270460128784, -0.936337947845459, -0.5937355756759644, -0.17254437506198883, 0.3037218153476715, -0.5084627270698547, -1.2898900508880615, -0.16709621250629425, -0.11040132492780685, 0.3945292830467224, 1.067285180091858, -0.6761237382888794, 0.9830290079116821, -0.9378317594528198, 1.0747040510177612, -1.0966960191726685, -0.040385521948337555, -0.30504441261291504, 0.717006504535675, 0.8453971147537231, 1.2823514938354492, -0.4510881006717682, 0.9310209155082703, 0.558989405632019, 1.7553545236587524, 0.22032733261585236, 0.34885087609291077, -0.49821725487709045, 0.6310348510742188, -0.005815248936414719, 1.7713183164596558, 0.6960068345069885, 1.0168753862380981, -0.4279497563838959, -0.3717988431453705, 0.3662603199481964, 0.5808044075965881, 0.2841570973396301, -0.5838310122489929, 0.9424929022789001, 0.49712178111076355, -1.8825936317443848, -0.38395166397094727, -0.08966327458620071, -0.19303108751773834, 0.0006350703188218176, -1.336133599281311, -0.47970229387283325, 0.7732596397399902, 0.34749361872673035, -0.8694649338722229, 0.056039515882730484, 0.614156186580658, -0.9861448407173157, 0.8249428272247314, 0.7251948714256287, -0.7107444405555725, 0.9408591389656067, 0.24639250338077545, -0.16263358294963837, -0.43706953525543213, -1.1554903984069824, -0.2629537284374237, 1.398998737335205, 0.49630436301231384, 0.46343663334846497, 1.454781174659729, -0.44522154331207275, -1.919171690940857, -0.6793184280395508, 1.3476675748825073, 0.7359772324562073, -2.0694470405578613, 0.331442266702652, 1.108801007270813, -0.16357500851154327, -1.6640782356262207, 0.49133211374282837, -0.48082220554351807, 1.4045593738555908, 0.41163763403892517, 0.6202586889266968, -1.7207363843917847, 0.02498026192188263, 0.8065261840820312, 1.0945310592651367, 0.36919543147087097, -0.45434844493865967, 0.8982983231544495, -0.3546108305454254, 1.141876459121704, -0.4245315194129944, -0.06065941974520683, 0.09911511093378067, -0.30670708417892456, -2.178581476211548, 0.04012543708086014, 0.3414766490459442, -0.6863856315612793, 0.14611677825450897, -0.2827599346637726, 0.7743133902549744, 0.40671634674072266, 1.3791100978851318, 0.7333961129188538, -0.5600520372390747, -0.5912978649139404, -0.02516036480665207, 1.2908825874328613, -0.20203442871570587, -0.1205342710018158, -0.45396938920021057, 0.3630537986755371, -0.5792959928512573, -1.2715932130813599, -1.0411449670791626, -0.07043391466140747, 0.3794000744819641, 0.961945116519928, -1.6532000303268433, 0.16288280487060547, 0.24973997473716736, -0.07585852593183517, -2.4619131088256836, -0.5288925170898438, -0.4729001820087433, 1.6074975728988647, -2.467520236968994, -2.026776075363159, 0.058209121227264404, -1.1628607511520386, -0.2531801164150238, 1.609282374382019, -0.15176908671855927, -0.06229072064161301, -1.2229355573654175, 0.03690073639154434, 1.7527588605880737, 0.05114532634615898, 0.49767807126045227, 1.0095525979995728, -0.5408116579055786, 0.8584250807762146, -0.5688663721084595, -2.3914198875427246, -0.6290503740310669, 0.558030366897583, 1.9310063123703003, 0.013090518303215504, 0.12453710287809372, -1.0546292066574097, -0.04040487855672836, -0.6122116446495056, 0.3939017057418823, -0.6760879755020142, 2.016597270965576, 0.38455435633659363, 0.2317303568124771, 0.261737585067749, 0.9931667447090149, 0.7720316052436829, -0.2680686414241791, 0.12233613431453705, -0.7588030099868774, -1.476785659790039, 2.3842713832855225, 0.1412825733423233, 0.49238285422325134, 0.055217768996953964, -0.3159047067165375, -0.9923238158226013, -0.35708320140838623, -0.4181957542896271, -1.1690105199813843, 0.354402631521225, 1.0218788385391235, -0.7987401485443115, 0.6895135641098022, -1.5400218963623047, -0.9216291308403015, 0.12818093597888947, -0.7946242690086365, -1.8038841485977173, -0.5249370336532593, 0.20001116394996643, 0.7253239750862122, -2.705897808074951, -0.6006414294242859, 0.7325475811958313, 0.13465215265750885, -1.9760653972625732, 0.30124738812446594, -1.2887977361679077, 0.3532550036907196, -0.5645192265510559, -1.8408817052841187, 0.18681471049785614, -0.5807203650474548, 2.0476953983306885, -0.542369544506073, -0.9496315121650696, 0.1934300661087036, 0.8125253915786743, 1.029571533203125, -0.9796624183654785, 1.8906272649765015, -0.4467793107032776, -0.15797562897205353, 2.055685043334961, 0.7589801549911499, -0.6550947427749634, -1.0402823686599731, 0.025563227012753487, 1.390629529953003, -0.47023385763168335, 1.421182632446289, -0.21363385021686554, -0.8543751835823059, -0.7918631434440613, -1.2075037956237793, -0.603108823299408, -0.8361235857009888, -0.7707242965698242, 0.9170895218849182, 1.307664394378662, 0.8484098315238953, -1.2290278673171997, 0.7473424673080444, 0.26119765639305115, 0.7233642339706421, 0.1998198926448822, 1.55048668384552, 0.849098265171051, 0.8545065522193909, -1.9293118715286255, -0.9477128982543945, -0.8046185970306396, 0.5440045595169067, 0.10518023371696472, -2.8286397457122803, 0.802602231502533, -0.3165193498134613, -0.08146537840366364, -1.5071196556091309, 1.0002906322479248, 0.283461332321167, 1.2569934129714966, -0.16542267799377441, 0.362992525100708, -0.37488266825675964, 0.47202807664871216, -0.2675226926803589, 2.2768807411193848, 1.3201831579208374, -0.19767425954341888, 0.6458160877227783, 0.15152031183242798, 1.024455189704895, -0.36312028765678406, 0.80363929271698, -0.39537250995635986, -2.245131492614746, -0.3582557737827301, 0.32620367407798767, 0.23266014456748962, -1.0563222169876099, -0.5212829113006592, -0.29667750000953674, 1.9234341382980347, 0.21887286007404327, 0.24951893091201782, -0.2527603805065155, 0.04796003922820091, 0.9321042895317078, 1.1112364530563354, 1.6124259233474731, -0.4626898169517517, -1.8887104988098145, 0.21814250946044922, -0.6064281463623047, -1.1004670858383179, 1.8191721439361572, -3.3219850063323975, 0.7216241955757141, 1.0375314950942993, 0.5598083138465881, -1.4659165143966675, -2.715491533279419, 0.6120033860206604, -0.003043548436835408, -0.9222621917724609, -2.368351936340332, -0.1436813920736313, 0.8746702075004578, -0.04081322252750397, -0.43864282965660095, -1.7204937934875488, 0.055077679455280304, 0.20117801427841187, 1.169254183769226, -0.44233644008636475, 1.7673943042755127, -0.8186147809028625, -0.30533039569854736, -1.1675976514816284, -0.16237857937812805, 0.4534721374511719, -0.23331716656684875, -0.09672278165817261, 0.9932478666305542, 1.0324158668518066, 0.271627813577652, 0.6095818877220154, 0.6992849111557007, 1.9572993516921997, -0.6329535245895386, -2.154784679412842, 0.045034345239400864, -1.9085866212844849, -0.3653852939605713, -0.5721405148506165, 0.1549878865480423, -0.523574709892273, 0.19328059256076813, 0.13651102781295776, -0.7765559554100037, -0.19389362633228302, -0.17433156073093414, 0.001759647740982473, 1.707889437675476, 0.10451941192150116, 0.4044601321220398, 1.434314489364624, 1.0392658710479736, 0.7746922373771667, -0.49949946999549866, 2.3786461353302, 0.45398086309432983, 0.09026502817869186, 0.3037203252315521, 0.07642146944999695, 0.4874263107776642, -0.31487879157066345, -0.25498878955841064, 1.3810018301010132, -0.6604655981063843, 0.24307355284690857, 1.4233585596084595, 0.24469080567359924, -0.2187015414237976, -1.464020848274231, 0.1668345183134079, 0.38977500796318054, -2.3948488235473633, -0.5711279511451721, -0.5328534245491028, -0.33298003673553467, -1.4743690490722656, -0.7544476985931396, 0.31198152899742126, 0.06926819682121277, -0.9175291657447815, -1.5051403045654297, 0.029513349756598473, 1.6767936944961548, 0.05654973164200783, -2.362138271331787, 0.6906533241271973, 0.4410509765148163, 0.24775433540344238, -0.7567102313041687, 1.6127711534500122, 1.4426158666610718, -0.2137206792831421, -2.3064334392547607, -1.641463041305542, 1.2538431882858276, 0.12292438000440598, 1.6626125574111938, -0.1525781899690628, 1.081170678138733, -0.9926771521568298, 0.3823223114013672, -1.0938690900802612, -0.9577855467796326, 0.7609598636627197, -1.6336851119995117, 0.8650490045547485, -3.1602470874786377, 0.4159798324108124, 0.3350899815559387, -0.4786502420902252, 0.5030946731567383, 0.7197094559669495, 0.47957655787467957, -0.37249019742012024, -0.8141595721244812, -0.8265440464019775, -0.2907715141773224, -0.2102339267730713, 0.2732146382331848, -0.15058082342147827, -0.9419897794723511, -0.4274406433105469, 0.8364837169647217, -1.5726730823516846, 1.0907710790634155, 2.730773448944092, 1.8656164407730103, -2.1263656616210938, -1.0130559206008911, 1.081437587738037, 0.4577925503253937, -1.3076400756835938, 2.1405017375946045, 1.5817186832427979, -0.8143823742866516, -1.0643959045410156, -0.23023024201393127, -0.2387441247701645, 1.966130018234253, 1.6146453619003296, -0.8331564664840698, -1.0334808826446533, -0.7978392839431763, -1.439943552017212, -1.3516172170639038, 0.035320866852998734, 0.025336137041449547, -0.07581306248903275, -1.078668236732483, 0.02970265969634056, 0.9072753190994263, 1.8013147115707397, -0.22572946548461914, -0.15720129013061523, 0.810151219367981, 0.45853671431541443, 0.41186070442199707, 2.4040451049804688, 0.4485790431499481, 0.06855981796979904, -0.6023041009902954, 0.15648886561393738, 0.4725379943847656, -0.0016873528948053718, 0.641599714756012, -1.3409796953201294, 0.8731667995452881, -1.0956772565841675, 0.042681317776441574, 1.4817836284637451, -0.3687528371810913, -0.5479604005813599, 0.7098768949508667, 1.5179498195648193, 0.44721531867980957, 1.3164758682250977, -0.072919100522995, 0.9112414121627808, -0.9840396046638489, -1.0805224180221558, 2.214543104171753, 0.14823408424854279, 0.11944643408060074, -2.20111083984375, 0.09100199490785599, -2.022062301635742, -0.2752384841442108, -0.48720601201057434, -0.6112703084945679, 0.8482052087783813, 0.9022803902626038, -1.3947436809539795, -0.9318264126777649, -0.5132947564125061, 0.5795575976371765, -0.6574289202690125, 1.1896936893463135, 0.16098466515541077, -0.16447852551937103, -0.4063660204410553, 0.5642118453979492, -2.2796945571899414, -2.0209600925445557, -0.19276922941207886, 0.781841516494751, -0.611395537853241, 0.5300466418266296, 1.503759741783142, 0.14670002460479736, 0.3443925678730011, -0.892004668712616, -0.10604963451623917, -1.291917324066162, -0.5830331444740295, 0.17016972601413727, -1.5785824060440063, -0.54850172996521, 0.3577828109264374, 1.340444803237915, -0.3853820860385895, 0.3648916780948639, 0.5884062647819519, -0.40042978525161743, -1.5863285064697266, 1.868098258972168, -0.4103979170322418, 0.09084677696228027, 0.6287230253219604, -0.889825701713562, 1.2655303478240967, 0.21628493070602417, -1.7120269536972046, -0.19424444437026978, -0.3844596743583679, -1.3204461336135864, 0.4851202666759491, 1.2884297370910645, -0.225113183259964, -0.40945008397102356, 1.3262449502944946, 1.5283324718475342, 1.5821119546890259, -0.23795737326145172, -1.657439947128296, 0.027467843145132065, 0.6851286292076111, 0.7391332983970642, 0.35355833172798157, 0.26817286014556885, 0.1231706365942955, 0.11925018578767776, 2.1064956188201904, -0.31247127056121826, -1.191117763519287, -1.5258595943450928, 0.08089781552553177, -2.1893248558044434, 1.0957294702529907, -0.39168891310691833, 1.3300094604492188, 0.8313359022140503, -1.135540246963501, 2.508000612258911, 0.8752701878547668, 0.6907079219818115, -0.2759682238101959, 0.08963396400213242, -0.7467788457870483, -0.44480931758880615, 1.4030969142913818, -0.7421023845672607, -0.0375741571187973, -0.4666871726512909, 0.6419519186019897, -0.30330702662467957, 0.23673376441001892, 0.27995142340660095, -0.1174260675907135, 0.6085219979286194, 0.24605779349803925, -0.5142150521278381, 0.5613433718681335, -0.31907010078430176, -0.5392710566520691, -0.37170788645744324, 0.2882237136363983, 0.7772586345672607, 0.28158190846443176, -0.2093001753091812, -0.19157804548740387, -2.0752739906311035, -1.6314254999160767, 1.235263466835022, 0.634280800819397, 1.4417648315429688, -0.2132113128900528, -1.7983438968658447, -1.765363097190857, 0.906337320804596, -0.07069722563028336, -1.3242570161819458, -0.012773897498846054, 0.8950173258781433, -0.2842195928096771, 1.290224313735962, -0.6006561517715454, 1.2379345893859863, -0.32792961597442627], [1.0915896892547607, 0.5707909464836121, 1.1854933500289917, 0.4215330183506012, 0.2988879382610321, -1.3975743055343628, -2.437868356704712, -2.107618808746338, -1.8261381387710571, 0.4960452616214752, 0.3788430392742157, 0.6712678670883179, -0.44938966631889343, -0.0614263191819191, -0.8031625747680664, -0.14485527575016022, 0.5828579068183899, -0.5760094523429871, -0.8680371642112732, 0.8279316425323486, -0.17821326851844788, 0.6144932508468628, -1.2075626850128174, -1.7125660181045532, -0.44124695658683777, -0.1819152534008026, -0.8086622953414917, 0.05666687339544296, 0.26106512546539307, -0.18718409538269043, -0.9057689309120178, -0.1270487755537033, -0.6355849504470825, -1.6492849588394165, 0.3096737861633301, 0.444690078496933, 1.2324182987213135, 0.42694321274757385, -1.1001287698745728, 0.6076736450195312, 1.1530965566635132, -1.8276530504226685, -0.06686484813690186, -0.6139768362045288, 0.33287930488586426, -1.2866814136505127, 1.3494411706924438, -0.737523078918457, 0.04411696270108223, -0.5968047380447388, 0.9027393460273743, -1.4383323192596436, -1.2112531661987305, -0.9248303174972534, 0.14464262127876282, 0.5008695721626282, 1.4653264284133911, 0.04072725027799606, 0.5562333464622498, 2.099794864654541, -0.3054025173187256, -1.5115678310394287, 0.7194522023200989, -0.44830310344696045, -1.5439188480377197, -0.7936501502990723, -0.5031991600990295, -0.21249336004257202, -0.14968307316303253, 1.3205723762512207, 1.6983516216278076, 0.1158691942691803, -0.3269592523574829, -0.3312426507472992, -0.576747715473175, -0.8635048270225525, -0.03853091225028038, 1.583930253982544, 1.4801661968231201, -0.5641859769821167, 0.2805539667606354, 0.4675462245941162, 0.43993204832077026, -1.1965782642364502, -0.07664714753627777, 1.0402125120162964, 0.5978029370307922, 1.3287657499313354, -1.0228201150894165, -1.1587936878204346, 0.3961621820926666, 0.615406334400177, 0.03029603883624077, 0.3418431878089905, 0.7729461789131165, -0.17605182528495789, 0.03681088611483574, 0.8229002356529236, -1.3184176683425903, 0.41351813077926636, -0.4140547811985016, -1.0989229679107666, 1.3336058855056763, -0.08326908200979233, 1.6811457872390747, 2.1850836277008057, 0.07979346066713333, 1.023486852645874, 0.4872923493385315, -0.44726869463920593, 0.4711851477622986, 0.460554838180542, -0.2585625648498535, -0.5701712369918823, -0.24944928288459778, 1.502415418624878, -0.035100918263196945, -0.32299867272377014, 1.4441546201705933, -0.719508707523346, -1.0690958499908447, -1.1483267545700073, -0.6973603963851929, -0.4449649453163147, -0.7746780514717102, 0.08460879325866699, 0.607866644859314, -1.1174298524856567, 0.090155690908432, -0.3879254460334778, 0.04595772176980972, 0.40911805629730225, 1.2364238500595093, 0.4617732763290405, -1.050297498703003, -0.2260752022266388, -0.7281870245933533, 0.3513634502887726, 0.9587352275848389, -0.8668474555015564, -2.1261415481567383, -0.3674994111061096, -0.6011212468147278, -0.42656850814819336, 0.9310208559036255, -0.2959812581539154, 1.590904951095581, -0.03501542657613754, 0.04171135649085045, -0.0023472511675208807, -0.26961880922317505, -0.500996470451355, -0.17065022885799408, -0.6515411734580994, -0.9456349015235901, 0.2049529105424881, -0.04763144999742508, 2.5813183784484863, 0.8894619941711426, 0.2045011967420578, 0.9402854442596436, -0.4033043682575226, 0.5265524387359619, 0.7728816866874695, 0.6047637462615967, -0.017463382333517075, -1.1911264657974243, -0.8074722290039062, 0.022617433220148087, -0.4156572222709656, 0.3095064163208008, 0.5487231612205505, -0.6142308712005615, -0.0324634313583374, -0.39713254570961, 1.7506853342056274, -2.0936338901519775, 0.9536739587783813, -1.1010938882827759, -1.9257550239562988, -2.0550646781921387, 0.40884363651275635, -3.4676690101623535, 0.7552384734153748, 0.4231557250022888, -2.3606460094451904, 0.15080071985721588, 1.7070719003677368, -0.7149297595024109, 1.2965468168258667, 0.13399741053581238, -0.21928702294826508, 0.8436129093170166, 0.29115694761276245, 1.82755708694458, -0.1681487262248993, 0.4539610743522644, 1.76353919506073, -0.3809278607368469, -0.36512672901153564, 1.0299922227859497, -0.02749742940068245, -0.6526944041252136, -0.34214046597480774, 0.32305312156677246, 0.7544234991073608, -0.8570680618286133, 0.13405324518680573, 0.3270484507083893, -0.04056239128112793, -0.8286926746368408, 1.0084930658340454, -0.25964537262916565, -1.1623432636260986, -0.2724588215351105, 0.9186293482780457, 1.2052428722381592, -1.4535552263259888, 0.7572286128997803, -0.973953127861023, 1.6142350435256958, 1.052289605140686, -0.09721186012029648, 0.164423868060112, -0.19624599814414978, -1.1783770322799683, 0.6818329691886902, -0.33895134925842285, -0.5239418745040894, 0.17619100213050842, -0.1506337672472, 0.41843360662460327, 0.4961668848991394, 0.2157309353351593, 0.8340035676956177, 0.8190608620643616, -1.2384048700332642, -0.01977315917611122, -2.1997344493865967, -0.6015124917030334, 0.9473405480384827, 0.9155652523040771, -1.8993752002716064, 0.5882050395011902, -0.42615360021591187, -1.2349282503128052, -0.2669212222099304, -2.1470675468444824, -0.4443853795528412, 0.8629996180534363, 1.8166348934173584, -0.3979600965976715, -0.7653948068618774, 0.8273773193359375, -0.9577299952507019, -0.4003002345561981, 0.6970629096031189, 1.759106993675232, -0.42546650767326355, 0.031092723831534386, -0.09649428725242615, -0.7104766964912415, 1.068491816520691, 0.29423636198043823, -1.6969107389450073, -0.11753679066896439, 0.7977720499038696, 2.5506997108459473, -0.10417665541172028, 0.5849573016166687, 0.7689868807792664, 0.4711962342262268, -0.2572239339351654, -0.20609699189662933, -0.28540652990341187, 0.8908764123916626, -1.047764778137207, -0.7999094724655151, 0.9920234084129333, 0.2729261815547943, -1.0599526166915894, 0.47444069385528564, 0.3568827211856842, 0.5332135558128357, -0.8925817608833313, -0.989974319934845, 0.34076017141342163, -0.12467335909605026, 0.034798599779605865, 1.238140344619751, -0.49376943707466125, -0.8659287095069885, -2.2580177783966064, -0.15283940732479095, 0.3735314607620239, -0.5442572236061096, -0.1740950047969818, 1.1977782249450684, 1.4568825960159302, 1.0945556163787842, 1.0590870380401611, -1.5185134410858154, 0.10508120059967041, -1.4812376499176025, -1.1395263671875, -0.46715816855430603, -0.8956440687179565, -0.8634746074676514, 1.0040876865386963, 0.5113932490348816, -0.41234901547431946, 0.6816397309303284, -1.3351640701293945, 0.4488350450992584, -0.2690657377243042, 1.1882349252700806, 1.2494311332702637, -1.8953266143798828, 0.16183814406394958, -1.0554463863372803, 1.063057541847229, -0.17772473394870758, -1.4916057586669922, 1.7954314947128296, 0.9903587698936462, -0.6745198369026184, -0.7988721132278442, -0.2719793915748596, -0.12995371222496033, 1.611028790473938, 0.23834124207496643, 1.4029279947280884, -1.7384065389633179, 1.2793080806732178, -1.680460810661316, -1.7804183959960938, 0.39950278401374817, -0.6429165005683899, 0.8709381818771362, 0.7489559650421143, 0.884284496307373, 1.9567402601242065, 1.5629122257232666, -1.5616663694381714, 1.5852004289627075, 0.4009891748428345, 0.42591843008995056, -1.036013126373291, 1.523991346359253, 0.00992942601442337, -1.4646358489990234, 0.20483896136283875, -0.18450364470481873, 1.8902021646499634, -0.596246063709259, -1.2747730016708374, -2.0095536708831787, 0.5873009562492371, 0.5975048542022705, -0.509064257144928, 0.11546291410923004, 0.33699914813041687, -0.6705771088600159, 1.0799552202224731, -0.19566796720027924, 0.755120038986206, -0.10498709231615067, -0.8258355259895325, -0.5276889204978943, -0.9668343663215637, -1.3595330715179443, -1.1410858631134033, -1.1376897096633911, -0.07560448348522186, 0.782455325126648, -1.2456040382385254, -0.7442178726196289, 1.2540088891983032, -0.2872002124786377, 0.6911658644676208, -0.13455228507518768, 1.1649208068847656, -0.9468335509300232, 0.5457600951194763, 0.5895482897758484, -1.681156039237976, 0.8686878681182861, -1.0981751680374146, -1.1244028806686401, -0.8412720561027527, 0.8478769659996033, 0.17013204097747803, -0.4193923771381378, -0.08566956967115402, -1.4193114042282104, 2.760263204574585, -0.8903350234031677, -0.8537966012954712, 0.910791277885437, -2.2453653812408447, 1.3721938133239746, 0.7559279203414917, -1.8655240535736084, -1.1560956239700317, -1.4453489780426025, -0.8376612067222595, 1.169662594795227, -0.5764771699905396, -1.965350866317749, 1.0365159511566162, -0.46096351742744446, 1.3427021503448486, 1.2350021600723267, -3.7102303504943848, -0.8161026239395142, -0.2542925477027893, -0.8865438103675842, -0.8034868240356445, -0.08145484328269958, -1.8574707508087158, -1.3597184419631958, 1.1539571285247803, -0.22185947000980377, 0.5315895080566406, -0.22357827425003052, 0.21149055659770966, 0.02654261700809002, 1.8467334508895874, 0.16225816309452057, 0.7384113669395447, -0.10965300351381302, -0.1513589471578598, 0.20504648983478546, 0.3665658235549927, -0.1463175266981125, 0.5907599329948425, 0.021972138434648514, 0.477485328912735, -0.9188398718833923, 1.195640206336975, -0.52997887134552, -0.6226074695587158, 1.1771036386489868, 0.7294248938560486, -1.5332796573638916, -2.3021557331085205, 1.4685804843902588, -0.17478451132774353, -0.29886558651924133, -0.31258219480514526, -1.0443696975708008, 2.0004312992095947, -1.3237162828445435, 0.9682326912879944, 1.0098711252212524, 0.27754494547843933, 0.37333962321281433, -0.4445904493331909, -0.3075323700904846, -1.0980271100997925, 0.10995480418205261, -1.2656208276748657, 0.2856229841709137, -0.4938064515590668, -0.14709769189357758, 0.08345311135053635, 1.2753608226776123, 0.6980496644973755, 1.7849668264389038, 2.5040929317474365, 2.7570278644561768, 0.15354382991790771, -1.1444008350372314, 0.2647446095943451, -0.029070068150758743, 1.185858130455017, -0.15241888165473938, -0.7687155604362488, 1.832318902015686, 0.6037852168083191, -1.7210911512374878, 0.20526345074176788, 0.0930609405040741, -1.1788092851638794, 2.122063159942627, 0.6829257011413574, -0.4598390460014343, 2.0989460945129395, 0.1936781257390976, 0.21435131132602692, 0.33657246828079224, 0.7193885445594788, 1.9196707010269165, -0.8921293616294861, -0.015475468710064888, 0.34881657361984253, -1.6418732404708862, -0.590652346611023, 0.43321165442466736, -0.3131280243396759, 0.5684893727302551, 0.18504701554775238, -0.004179414827376604, -0.07094991952180862, -2.1746573448181152, -1.0825623273849487, 0.557768702507019, -0.15593355894088745, 0.675865650177002, 0.16441668570041656, -0.2681218981742859, -0.8378875851631165, -0.8880888819694519, 2.015298366546631, 1.8692302703857422, -0.10922950506210327, 0.5991136431694031, -1.4157840013504028, 0.9872716069221497, 1.5406919717788696, -0.49333569407463074, -0.10319552570581436, -0.8850009441375732, 0.91539067029953, 0.05067161098122597, -1.369766354560852, -1.5331072807312012, -0.10041407495737076, 0.7450506687164307, 0.8060261011123657, 0.6830964684486389, -1.3250783681869507, 0.8818110227584839, 1.7086118459701538, -1.811779499053955, 0.20664073526859283, -0.333131343126297, 1.250220775604248, 0.11729355901479721, -0.514864981174469, -1.181747317314148, 0.5308250188827515, 0.7083812355995178, -0.1817752569913864, -0.8475263714790344, -1.801795244216919, -0.3561762571334839, 0.7585117220878601, -0.22704118490219116, -0.35117536783218384, -0.44723185896873474, -0.1105397567152977, -0.6578831672668457, -0.2960687577724457, 0.762891411781311, 0.36821192502975464, -0.23077493906021118, 0.2977081537246704, 1.6120977401733398, -1.3692289590835571, 0.07012420892715454, -0.6568984985351562, -0.5013191103935242, -0.4684208333492279, -1.0077769756317139, -0.4454321265220642, -0.15967155992984772, -0.11731688678264618, -1.8637328147888184, 0.22086912393569946, -0.013259731233119965, -0.49924325942993164, 1.38676917552948, -1.9916632175445557, 0.06150176376104355, 0.040046919137239456, -0.625740647315979, -2.0415046215057373, 0.5242916345596313, 0.0862591490149498, 0.10124091058969498, -0.2551295757293701, -1.7664966583251953, -1.0154873132705688, -1.8165820837020874, -0.3730565905570984, -0.6164369583129883, 0.9822953939437866, 1.362125277519226, -1.0155097246170044, -0.8921962976455688, -1.6830801963806152, -1.9827609062194824, -0.6591919660568237, -0.7415086627006531, 1.049872875213623, 0.2393592745065689, -1.7829036712646484, -0.23685599863529205, 0.6145862936973572, -0.6589195132255554, 0.10578480362892151, 0.6948516964912415, -1.6785460710525513, -1.0841909646987915, -0.7816252112388611, -0.3311193287372589, -1.2925095558166504, -0.09037689864635468, -1.3575470447540283, 0.38790443539619446, -0.1550699919462204, 1.0988132953643799, 0.5379671454429626, 1.6502964496612549, -1.334241271018982, -0.37496834993362427, -0.1311645805835724, -1.2085628509521484, -1.369672179222107, -0.8349242806434631, 0.3190912902355194, 1.8987541198730469, -0.2199554294347763, 0.8305531740188599, 0.2625088393688202, -0.2377556562423706, -0.19167305529117584, -0.9233489036560059, 0.29670143127441406, -0.09306955337524414, 1.3692377805709839, 1.089967131614685, -0.09322584420442581, 1.1861076354980469, 1.7152483463287354, -0.011826836504042149, -2.6898860931396484, -1.6325099468231201, -1.9627983570098877, 3.374176025390625, 0.5487720370292664, 0.9278536438941956, -0.9779964685440063, -0.1400606632232666, 1.141278624534607, -1.1841063499450684, -1.3602378368377686, 0.929920494556427, 0.1620994657278061, 0.340695321559906, 1.314092755317688, 1.5603712797164917, 0.8023707866668701, 0.1187555119395256, -0.22825482487678528, 0.9088571667671204, -0.6535536646842957, 2.0699210166931152, 0.45061758160591125, -1.3558292388916016, 0.3915283679962158, -0.8124152421951294, -0.5408315062522888, -1.9180717468261719, -0.6101540923118591, 2.232835531234741, -0.0827692374587059, -0.8040947914123535, 1.458505392074585, 0.8482280969619751, 0.7935416102409363, 0.2804258167743683, 0.3260611295700073, 0.3364189863204956, 0.9789747595787048, -1.2472351789474487, -1.5707347393035889, -0.23206108808517456, -0.7863492965698242, 0.01641738973557949, -0.11771796643733978, -1.919212818145752, -0.4878275692462921, 0.15831637382507324, -0.08459199219942093, -0.762774646282196, 1.5653619766235352, 0.32661473751068115, 0.19013527035713196, 0.8950775861740112, -0.5620213747024536, -0.7741967439651489, -1.3063499927520752, -2.0743467807769775, 2.196171283721924, 0.2083979994058609, 1.2630231380462646, -1.8139605522155762, -0.1950933039188385, 0.8773106336593628, 0.8935256600379944, 0.507544994354248, -1.3464919328689575, -0.6279036402702332, 1.4799779653549194, 0.532950758934021, 1.067036747932434, -0.46533507108688354, 0.5221546292304993, -0.6302579641342163, 0.6688107848167419, -0.6515815258026123, -0.7774897813796997, 0.026155130937695503, -0.7541631460189819, -0.9711080193519592, 1.5283247232437134, -1.2558690309524536, -0.8559820652008057, 0.3585110008716583, -0.778984785079956, -0.15856482088565826, 1.156117558479309, 0.9316028952598572, -0.35328418016433716, -1.399317979812622, -0.15746940672397614, 0.6872553825378418, 0.1921885758638382, 0.07680913060903549, -1.0453901290893555, 0.28810247778892517, 2.4661202430725098, 0.3153938055038452, 0.21579860150814056, 0.8795116543769836, 1.285919189453125, -1.016534447669983, -0.31432586908340454, -1.405525803565979, -0.030929332599043846, -0.003476519137620926, -1.356948971748352, -1.379056453704834, 0.4399312138557434, -0.8879655003547668, -0.7528440952301025, -0.5799059271812439, -0.2537226378917694, -0.6550247073173523, 0.7378858327865601, -1.3912909030914307, 1.2124888896942139, -1.0898243188858032, 0.05382300168275833, -0.38061803579330444, -2.1502745151519775, 0.24841554462909698, -1.7217355966567993, 1.8920674324035645, 0.7784862518310547, -0.17745959758758545, 0.3475448787212372, 1.4093290567398071, -1.41209876537323, -0.02474423497915268, -0.04596949741244316, -0.02349645458161831, -1.096685767173767, -0.29027581214904785, 0.09793064743280411, -0.30080801248550415, 0.031397562474012375, 0.35171911120414734, -0.13690458238124847, -0.864781379699707, 1.223068356513977, 1.505340337753296, -0.5324687957763672, -0.38873225450515747, -0.28970953822135925, 0.20753952860832214, -0.1610441356897354, -0.43978244066238403, -0.9841373562812805, -1.0726897716522217, -1.179388403892517, -1.0345609188079834, 0.2511592209339142, -0.3388197720050812, -1.2797987461090088, 0.6500148177146912, -0.9306641817092896, -0.1555710881948471, -0.18185530602931976, 0.5098899602890015, -0.00682267639786005, 0.09269387274980545, -0.3653951585292816, 1.7177141904830933, 1.0931819677352905, 0.22400961816310883, -0.6091060042381287, -0.3402789235115051, 1.7673043012619019, -0.36504143476486206, 0.76875239610672, 0.49900132417678833, 0.7072482109069824, 0.6974075436592102, 0.5745273232460022, -0.24269907176494598, -1.8844013214111328, -1.4429057836532593, 0.45832812786102295, 0.6421666741371155, -0.681863009929657, -0.7756879925727844, -0.43166330456733704, 0.25492557883262634, 0.47064468264579773, -1.0038810968399048, 2.054414749145508, 1.1490079164505005, -0.7129355669021606, 0.5696709156036377, 0.7751395106315613, -0.6957831978797913, 1.2888582944869995, -0.34048253297805786, -1.8641142845153809, 1.6914808750152588, 1.1513210535049438, -0.8733839392662048, -0.44474101066589355, 0.43377989530563354, 0.3465655744075775, 0.475140780210495, -0.8718868494033813, 0.783057451248169, -1.2746939659118652, 0.10391000658273697, -0.2272343635559082, -0.11466286331415176, 1.3268277645111084, -1.21619713306427, -0.19433346390724182, 0.8581743240356445, -0.6701979637145996, 0.39899957180023193, -3.1144983768463135, 2.5845861434936523, -1.3249915838241577, 1.2075330018997192, -0.8871645331382751, -0.7397716641426086, -0.07745523005723953, -0.20064252614974976, -0.16589757800102234, 0.01620335318148136, 0.05728769302368164, -1.210547685623169, -1.6397596597671509, 0.7045593857765198, 0.7738115787506104, -0.4005524516105652, -0.5661526322364807, -0.1634955257177353, 0.6094635128974915, 0.2703096866607666, -0.1732865273952484, 1.5388067960739136, -1.81378173828125, -2.8818750381469727, 0.9237762093544006, 1.5763261318206787, -1.1006033420562744, 1.3006755113601685, -1.0317142009735107, 0.7917214035987854, -0.33335080742836, -0.9605067372322083, -1.428140640258789, -0.8717836141586304, -0.9255107641220093, 0.6103534698486328, -0.002235343214124441, -1.275965690612793, 0.8538976907730103, 0.9654603600502014, -0.008486256003379822, -0.9158895015716553, 0.4622648358345032, -2.3627007007598877, -0.9751567244529724, 0.039371557533741, -0.6418896317481995, -1.1524792909622192, -0.3558422923088074, 1.1746470928192139, 0.7179595232009888, 0.9473751783370972, -0.9695817828178406, 1.2082217931747437, -1.9522937536239624, 0.18630574643611908, -0.3368903696537018, -1.901391863822937, -0.09619124978780746, 1.563079833984375, 1.4679220914840698, -0.8845681548118591, 0.2652308940887451, 0.6525386571884155, 0.08149777352809906, 1.0586005449295044, -0.19985538721084595, 2.003493070602417, 2.295689582824707, 1.6883766651153564, -1.8983534574508667, 0.8607844114303589, 0.3314460813999176, 0.2838410437107086, -1.4295589923858643, 1.780880331993103, -1.8158725500106812, 0.5830557942390442, -0.3626725375652313, 1.2676769495010376, 0.05259402468800545, 0.5876591205596924, -0.0025590509176254272, 0.9823637008666992, -0.6683188080787659, 1.068088173866272, 0.9319712519645691, 0.8654567003250122, -0.3118259608745575, -0.2600942552089691, -0.023605141788721085, 0.9949472546577454, 0.34915801882743835, -0.8561790585517883, 2.4242136478424072, 0.5738168358802795, -0.22147059440612793, -0.6261398196220398, 0.9660587310791016, 0.724410355091095, -0.375044047832489, 0.5585289597511292, 1.9461536407470703, 0.6860015392303467, 1.7169750928878784, -0.32428571581840515, 0.5016548037528992, -0.8326855897903442, -1.9103407859802246, 1.2296431064605713, -0.6309681534767151, 0.1918226182460785, -1.4750484228134155, 0.6886651515960693, -0.40755245089530945, -0.6751216650009155, -0.7266293168067932, 0.9046347141265869, -0.3953905999660492, 0.19827406108379364, 0.5196812748908997, 0.769627571105957, -0.8253910541534424, -1.6697953939437866, -0.15255510807037354, -1.1395106315612793, 0.2358977198600769, 0.6010527014732361, 0.41911786794662476, 0.19759979844093323, -0.9035012125968933, 0.2894127070903778, -0.04106292873620987, 1.9707081317901611, -1.1115529537200928, -2.253856897354126, -0.23959369957447052, 1.5600337982177734, -0.5055134892463684, 0.3944435715675354, 0.07027899473905563, -0.5145622491836548, -1.4866260290145874, 1.433733582496643, 1.165411114692688, 0.5015652179718018, 0.007855668663978577, -0.7095927000045776, -2.5724246501922607, -1.0857455730438232, 0.3729225695133209, 0.8695487380027771, -0.10018764436244965, 1.4450982809066772, -0.9013293385505676, 1.350521206855774, -0.12752488255500793, 0.8448728322982788, -0.052818600088357925, -0.2534903883934021, -0.811187744140625, -2.4506869316101074, -0.38160502910614014, 0.6501498818397522, 1.092100977897644, -0.20240867137908936, 1.5780308246612549, -1.4368213415145874, -2.09831166267395, 0.8224117755889893, -0.330581933259964, 0.2163141369819641, -1.534092664718628], [0.4922747313976288, 0.7644490599632263, 0.4097953140735626, -0.8001607656478882, 1.4802653789520264, -0.3792310357093811, -1.7233484983444214, 1.5356712341308594, 0.18101711571216583, -0.05430515110492706, 1.7491499185562134, -0.28991177678108215, -1.2333807945251465, 0.374864786863327, -0.15734560787677765, -1.3222885131835938, 1.1108481884002686, 1.014344334602356, 0.24675872921943665, 0.17735019326210022, 1.230223536491394, 0.011609855107963085, 0.042079661041498184, 0.5124095678329468, -0.9538488984107971, 1.710148572921753, -0.2011888027191162, -1.904193639755249, -0.9076153039932251, 0.42972320318222046, -0.340238481760025, 0.7644264698028564, 0.6600045561790466, 0.7525323033332825, -0.10076235979795456, -0.23135776817798615, 0.7048845887184143, 0.7114040851593018, -0.5804116129875183, 1.26963472366333, -0.1872563362121582, -0.7784876823425293, 0.788454532623291, 0.9631995558738708, -1.2745431661605835, 0.06724335253238678, -0.6858872771263123, -1.0509663820266724, -1.5853822231292725, -0.36776331067085266, -0.8486252427101135, 0.9662438035011292, -0.7125042676925659, -0.49997642636299133, -0.7477737069129944, 0.017428316175937653, 0.3964419364929199, 1.1996368169784546, 0.41511115431785583, -0.6495513319969177, -0.19774357974529266, -0.19215621054172516, 1.5180714130401611, 1.3885618448257446, 0.7297916412353516, 0.46815216541290283, 1.1184989213943481, -0.24547618627548218, -0.12005670368671417, -1.698608636856079, 0.4294203817844391, -0.48686420917510986, -0.3825342357158661, -1.2378631830215454, 1.6222636699676514, -1.2109959125518799, 1.02311110496521, -0.1058584600687027, 0.06103108078241348, -0.6716440916061401, 2.3881778717041016, 0.24459630250930786, 0.060393042862415314, -1.4238606691360474, 1.717107892036438, -1.7605282068252563, -1.7838515043258667, 0.18689508736133575, 0.7385230660438538, 0.9428578615188599, 0.5598146915435791, -1.133103847503662, -0.8690937757492065, 1.5646260976791382, -1.3705273866653442, 0.7683361172676086, -0.5862768888473511, 0.40536144375801086, -0.07960242033004761, 0.3613697290420532, -0.4261051118373871, -0.4958862066268921, -0.0390530601143837, -0.530412495136261, -2.034921169281006, -1.03286874294281, -1.3136014938354492, 0.19585222005844116, -0.1104109063744545, 0.034681886434555054, -1.2480287551879883, 0.49010947346687317, -0.27840539813041687, -1.8287566900253296, 0.024015910923480988, 0.5780446529388428, -0.47587645053863525, 0.6698701977729797, -0.4034026563167572, -0.15232497453689575, -1.2619153261184692, -0.8118648529052734, 0.15831215679645538, -1.6318018436431885, 0.02675589732825756, 0.15118403732776642, -0.2647535800933838, -1.002986192703247, 1.3265776634216309, -0.7447653412818909, 0.2706954777240753, -0.15027885138988495, -1.3342103958129883, 0.33796077966690063, -0.8461328148841858, 0.04708115756511688, 0.17507220804691315, -0.5240542888641357, 0.8155588507652283, 0.38503262400627136, -1.5534493923187256, -0.1746186465024948, -0.6589926481246948, 0.9016222357749939, -0.05000246316194534, -1.1209243535995483, 0.7754915952682495, 1.1304831504821777, -0.24555644392967224, 0.11268604546785355, -0.5432573556900024, -0.47370877861976624, 0.8698011636734009, 0.43082478642463684, 0.6708190441131592, -0.9018334746360779, -0.18475644290447235, 0.8033660650253296, 1.3497869968414307, 0.22433064877986908, -1.0633344650268555, 0.6455782055854797, -0.23417100310325623, 1.5428544282913208, -0.09071929007768631, 0.6473246216773987, 1.237001895904541, 1.1144165992736816, -0.6045450568199158, -0.9275316596031189, 0.7216867804527283, -0.6003393530845642, -0.597175657749176, 2.179128885269165, 0.6373162865638733, 0.7035639882087708, 1.7527095079421997, -0.2770536243915558, -0.5220445394515991, -2.1611263751983643, -0.11374697089195251, 1.2825626134872437, -0.18946875631809235, -1.887447714805603, 0.7309721112251282, -0.46032676100730896, 0.9026144742965698, 2.504103899002075, -0.8793194890022278, 0.607723593711853, -0.5819496512413025, 2.0215775966644287, 1.182781457901001, 0.34340742230415344, 1.57467520236969, -0.8136334419250488, 2.568828582763672, -1.8319203853607178, -0.38497045636177063, -1.2039155960083008, -0.31579744815826416, -0.5378186106681824, 0.2973004877567291, 0.2719956040382385, -0.7807724475860596, 1.6501290798187256, -0.35707637667655945, -1.3535187244415283, 0.3708358407020569, -1.2580616474151611, -1.5043679475784302, 1.856791377067566, 0.8491230010986328, 1.7424829006195068, 0.43811842799186707, -0.9226030707359314, 1.9985212087631226, 0.3909890651702881, 1.307617425918579, 0.6491541266441345, 1.0868791341781616, 0.033767811954021454, -1.345031499862671, 1.626410961151123, 0.5209323167800903, 1.7063510417938232, -1.1599928140640259, 0.727348804473877, 1.1249703168869019, 1.3349734544754028, -0.6758297681808472, -0.21627908945083618, -1.3191568851470947, 1.7336279153823853, 0.7834082841873169, -0.6731540560722351, -0.7702189683914185, 0.256059467792511, 0.5890293717384338, -1.9526203870773315, -0.21939368546009064, 0.26018980145454407, 0.31305399537086487, 1.0592498779296875, -1.0598409175872803, -2.6565260887145996, -0.5736469030380249, 0.9351016283035278, 1.257307767868042, -1.935726284980774, -0.47556835412979126, -0.009652731940150261, -1.1032980680465698, -0.25612443685531616, 0.11798425763845444, 0.4267427623271942, 2.1748759746551514, 0.07366599887609482, -1.7240610122680664, 0.07499846816062927, 0.4533880352973938, -0.22007733583450317, -0.49002113938331604, 0.03994244709610939, 0.6047000885009766, -1.142566204071045, 1.3542431592941284, -0.00312986527569592, -0.013394765555858612, 1.3808611631393433, -0.8659857511520386, -0.03975919261574745, -0.23583655059337616, 0.05560817942023277, 0.062368061393499374, 1.6195281744003296, -0.3011969327926636, -0.9469066262245178, 0.759415864944458, 1.8176475763320923, 0.883574903011322, 0.353065550327301, -0.6428905725479126, -1.8163748979568481, 1.0549453496932983, 0.25802960991859436, 0.7426356673240662, 0.37149694561958313, 0.1312127411365509, 1.1644785404205322, 0.7757406830787659, 0.25558722019195557, -0.011133654974400997, 0.02293984405696392, 1.8156592845916748, -0.0767575055360794, 0.8452605605125427, -1.4286887645721436, 1.0568370819091797, 0.6473216414451599, -0.13526339828968048, 0.32558584213256836, 0.5907766819000244, -2.2497706413269043, 0.4891924560070038, 1.2894083261489868, 0.4393825829029083, -1.3521335124969482, 0.2541893720626831, 0.4172880947589874, -1.631288766860962, 0.7519425749778748, -1.3531852960586548, -1.3191536664962769, 0.5627622604370117, 1.037670612335205, -1.2354196310043335, -1.1362000703811646, -0.4714527428150177, 1.878299593925476, -0.08458323031663895, 1.657578945159912, 2.160707473754883, 2.5987892150878906, 0.7632603049278259, 1.425675392150879, 1.33049476146698, 0.17763268947601318, 1.120136022567749, -0.48267918825149536, 0.9007540941238403, 1.69080650806427, -2.9166364669799805, 0.7282038927078247, -0.04221128299832344, -0.5532166957855225, 0.5451593399047852, 3.6507725715637207, -0.4841379225254059, -1.9822111129760742, -0.7415089011192322, -1.7512586116790771, 0.5719298720359802, -0.5921483635902405, -0.4316026270389557, 0.6128529906272888, 0.34313514828681946, 0.9403060674667358, -1.5974191427230835, -1.14795982837677, -0.3731388449668884, 1.4497158527374268, -0.19368772208690643, 0.9135533571243286, -0.4755224883556366, 0.35383161902427673, 1.9805842638015747, -0.17598168551921844, -0.05390707030892372, -0.5875115990638733, -1.2428725957870483, -1.8535583019256592, -2.0474019050598145, 0.15618003904819489, 0.4014461040496826, 0.4853450655937195, -0.2810070514678955, -0.15010154247283936, -0.45073986053466797, 0.43812355399131775, -0.9169093370437622, 0.1927516609430313, 0.16607791185379028, 0.3038314878940582, 0.6810817122459412, -0.3395788073539734, -0.17145465314388275, -0.2636268734931946, 0.33182480931282043, 0.13961578905582428, -0.8546479940414429, 0.9802380800247192, 0.3609543442726135, -0.355249285697937, 0.19470088183879852, 0.6015918850898743, 0.7351059913635254, 0.27120241522789, -0.37820836901664734, -1.1153916120529175, -0.38599053025245667, -0.3928350508213043, -0.12151020765304565, 0.3540777862071991, 0.3397194445133209, -1.8341929912567139, 0.8603857159614563, -0.44831693172454834, -1.4295223951339722, -0.9133656620979309, -0.4101937413215637, 0.3290364146232605, -0.5523114204406738, -0.945902407169342, -0.9163034558296204, -0.6868565678596497, -0.0768917128443718, 2.0115373134613037, 1.9162611961364746, -1.0337166786193848, 0.11118250340223312, 0.6976016759872437, 0.20479151606559753, 0.43804052472114563, 0.5694671273231506, -1.1032438278198242, 2.716153144836426, 0.997722327709198, -1.0816857814788818, -1.4825904369354248, 0.4930412173271179, 0.7125231027603149, -0.10179083794355392, 0.6250618100166321, 0.3295382261276245, -0.6293284296989441, -0.7626961469650269, 1.89348566532135, 0.6197676062583923, -0.16273604333400726, -2.164487838745117, 0.5224612951278687, -0.23800861835479736, -0.012320831418037415, 1.1955147981643677, -0.11966826766729355, 0.8682123422622681, -0.8295810222625732, 1.4811546802520752, 0.28775128722190857, -1.1144472360610962, 0.4003521502017975, 0.8655949831008911, 1.0637285709381104, -0.8565469980239868, -0.9766907691955566, 1.9056452512741089, -0.31190061569213867, 0.06448963284492493, -3.0644657611846924, 1.1863144636154175, -1.212796926498413, -1.9354218244552612, -1.2335957288742065, 1.0106581449508667, -1.2877672910690308, -0.7363710999488831, -0.7592142224311829, -0.8546868562698364, 0.7149280905723572, 0.347260445356369, -1.0140795707702637, -0.24683980643749237, -2.2871193885803223, -0.07072959840297699, 0.9721549153327942, -0.5451287627220154, 0.2372438907623291, 0.440748006105423, -1.0675243139266968, -0.311093807220459, 1.106814980506897, 2.4047486782073975, 0.7108300924301147, -0.8241112232208252, 0.6431417465209961, 0.2545839846134186, 0.7940785884857178, 0.05503139644861221, -0.20521922409534454, 1.6265153884887695, 0.43416786193847656, 0.5921057462692261, -1.2456108331680298, -1.7067874670028687, -0.8849174976348877, -1.2164684534072876, -0.6736825108528137, -1.0210425853729248, 1.0759773254394531, 0.04765184223651886, 0.32502487301826477, -0.5584149956703186, -0.2060486525297165, -1.368353009223938, -0.07057590782642365, 2.475825071334839, -0.14266176521778107, -0.4060969948768616, -0.8282724618911743, 0.18651585280895233, 1.0719904899597168, -0.7941192388534546, -2.1614813804626465, -0.12761780619621277, -0.3389147222042084, -0.9137412905693054, -0.2743256390094757, -0.236475870013237, 0.997441291809082, -0.6288385987281799, -0.2863416373729706, -0.6550431251525879, -0.00857265293598175, 2.6704306602478027, -1.0385279655456543, -1.4380624294281006, 1.5440882444381714, -0.38698866963386536, -0.959212064743042, 0.3206322193145752, 0.038103897124528885, 0.32765287160873413, -0.07435839623212814, -1.8150863647460938, -0.6380015015602112, 1.9087774753570557, 0.8239565491676331, 0.8971234560012817, -2.5302212238311768, -0.7300454378128052, 0.04699360206723213, 0.2680354118347168, -1.5168567895889282, 0.38062697649002075, 0.5210658311843872, -0.6444360017776489, -0.13004063069820404, 0.7545384764671326, 0.4491689205169678, 0.3914245367050171, -0.4816782772541046, 1.4771203994750977, -0.3678852617740631, 0.3545302748680115, 1.2238972187042236, -0.33932602405548096, -1.7988650798797607, -1.5794764757156372, 2.54128098487854, -1.2799474000930786, -1.8448913097381592, -0.7457665205001831, -0.4909249544143677, -0.1995985060930252, -0.5223405957221985, 1.0691807270050049, 0.7840786576271057, 1.084583044052124, -0.6360715627670288, 0.19359125196933746, -0.30491024255752563, -0.2752285301685333, -0.327006459236145, 2.0210912227630615, 1.0178011655807495, -0.18756094574928284, 1.1900047063827515, -0.16386578977108002, -1.634071707725525, -0.36926648020744324, 0.19232335686683655, 1.5105780363082886, 0.41828230023384094, -1.0714749097824097, 0.8732439875602722, -1.6323139667510986, -1.4199405908584595, -0.0391550287604332, 0.3200002908706665, 1.0397971868515015, -1.4888159036636353, -0.24912871420383453, -1.775994062423706, 1.6901612281799316, 0.13651271164417267, 1.3069270849227905, -1.690650224685669, 1.8393542766571045, -0.17459283769130707, 0.3426656126976013, -1.0222853422164917, -0.8450261354446411, 0.3391859829425812, 0.5027997493743896, 0.8728966116905212, 0.32865527272224426, 1.4306411743164062, 0.8722496032714844, -0.8349437713623047, 0.8652670383453369, -1.0827137231826782, -1.1816656589508057, 0.8296152949333191, 0.8099557757377625, 0.8341964483261108, -1.452493667602539, -2.0727248191833496, -0.8262345194816589, -0.34373271465301514, 1.8032385110855103, 0.7509086728096008, 0.4093978703022003, 0.3498139977455139, 0.09734878689050674, 0.2085818201303482, -1.8944660425186157, -0.5330643653869629, 0.25653275847435, -0.19030001759529114, 0.3991142809391022, -0.6215746402740479, -1.1673052310943604, -1.9556047916412354, -1.8897438049316406, -0.3365074098110199, 0.19472533464431763, 0.6272969841957092, 0.2915217876434326, 0.2617018520832062, -1.6056772470474243, 0.6087104082107544, 1.1154866218566895, 1.3539451360702515, -1.648919939994812, 0.5481728911399841, 2.157169818878174, 1.6228852272033691, 1.5614545345306396, 0.46591347455978394, -0.03131586313247681, -0.27357879281044006, -1.4431710243225098, -1.1222739219665527, 1.69429612159729, 1.0618362426757812, 1.7823611497879028, 0.955315113067627, -0.09363802522420883, -0.5064691305160522, -0.9622479677200317, 0.01290357206016779, 0.4088621139526367, 0.1270611733198166, -1.1557055711746216, 1.298604130744934, 0.10705774277448654, -0.6114056706428528, 0.6857008934020996, 0.09715069085359573, 1.0327768325805664, 0.6410675048828125, -1.7468351125717163, 0.6696125864982605, -0.9882590770721436, 0.5018864870071411, -0.9895528554916382, -0.8161964416503906, 0.6738235354423523, 0.918354332447052, 0.1338256448507309, 1.1674977540969849, -0.25819581747055054, -0.06674788147211075, -0.5969160199165344, 0.25314658880233765, -0.5472757816314697, -0.6058745384216309, 0.7773266434669495, -0.363372266292572, -0.07160763442516327, 0.5985751748085022, 0.41522201895713806, 0.8638277649879456, 0.3838483989238739, -0.9892932772636414, 0.4048992693424225, 0.9858617782592773, -0.07333824038505554, 0.40418580174446106, -0.41152769327163696, -2.0338776111602783, -1.1183850765228271, 0.45340827107429504, 1.037422776222229, -1.2805005311965942, -0.36234745383262634, -0.3531443774700165, 0.2689549922943115, -0.4711276888847351, 0.8809170126914978, 0.38569217920303345, -0.3478235900402069, 0.37527018785476685, -0.3133164048194885, -0.15298447012901306, -1.0812082290649414, -0.6409440636634827, 0.27234947681427, -0.4838198125362396, 0.0854245200753212, -1.0552911758422852, 1.1190520524978638, -0.18997332453727722, -0.29395419359207153, -0.5054835677146912, 0.9981414675712585, 2.3761372566223145, 1.6315600872039795, -0.36702263355255127, -0.8459284901618958, 1.1481399536132812, 0.009469250217080116, -0.6298728585243225, -0.45431604981422424, 0.18932965397834778, 0.8472543954849243, -0.005643734708428383, -0.5209783911705017, 0.5410004258155823, -0.13833117485046387, 0.022445183247327805, -1.270725965499878, 0.7720686793327332, 0.7554506659507751, -0.5253534317016602, 1.6360256671905518, 0.43415722250938416, -0.6672077178955078, -0.9310633540153503, -0.257668673992157, 1.5209271907806396, 0.47647953033447266, -0.5605210065841675, -1.2335362434387207, -0.9206835627555847, 0.548175573348999, -0.522017240524292, -0.25927960872650146, 0.8626280426979065, 0.3427906930446625, 0.12490975856781006, 0.29586800932884216, -0.327890545129776, -0.9678292870521545, 1.0068261623382568, -0.409390926361084, -1.3297436237335205, -0.13667723536491394, 0.3043631911277771, 0.4177226424217224, 1.2562531232833862, -0.7316213846206665, 0.5128061771392822, -0.13005048036575317, 1.5787158012390137, -0.9820066690444946, -0.40165019035339355, 1.661954641342163, -0.8428106904029846, 1.512068510055542, -1.12997567653656, -0.5518483519554138, 1.161592960357666, -0.03522982448339462, -0.13958071172237396, -0.4076191484928131, -0.35266244411468506, -0.2380380928516388, 0.8285205960273743, -1.029788613319397, 0.0702725350856781, 0.14523591101169586, 0.3473101556301117, -0.4293481409549713, 0.21963004767894745, 0.4325259029865265, 0.4398808777332306, 0.17005716264247894, -0.4824298322200775, -1.0606794357299805, 0.6045830249786377, 0.041149284690618515, -2.7428691387176514, 1.0556111335754395, -1.856725811958313, 0.05424942076206207, 0.6691275238990784, 0.42410358786582947, -1.4429327249526978, -0.06703734397888184, -1.5856506824493408, 0.273004412651062, -1.196527361869812, 0.6159067749977112, 0.834766685962677, 0.2190743088722229, -1.5553885698318481, 0.7816756367683411, 1.2765922546386719, 0.5571550130844116, 0.22795100510120392, 0.22861920297145844, 0.27290257811546326, 0.9501161575317383, 1.565495252609253, 0.7847169637680054, -0.1565340906381607, -1.496991753578186, 2.181882619857788, 0.13214266300201416, -0.6607075929641724, -0.4401506781578064, -0.08592677116394043, 1.3971247673034668, -1.382083535194397, -0.22600172460079193, 1.327901840209961, 0.158023402094841, 0.3714643120765686, 0.11508194357156754, -0.20396600663661957, -0.13163651525974274, 0.28386789560317993, 0.9772928357124329, -1.234136700630188, -1.2682371139526367, 0.1963437795639038, 2.239082098007202, 0.6053054928779602, -0.020372238010168076, -0.9268760681152344, 0.38972580432891846, -1.371669054031372, 1.7756731510162354, 0.3188899755477905, 0.07932443916797638, 0.1457655131816864, 0.3260462284088135, -0.10003738850355148, 0.8166506290435791, -1.048287272453308, -3.0263731479644775, 0.2777506411075592, -1.1768467426300049, 0.7097370028495789, 0.5296975374221802, -1.7515336275100708, 1.582972764968872, -1.1577916145324707, -1.062665581703186, 0.30450940132141113, 1.0440082550048828, 0.9209200739860535, -0.47126176953315735, -0.40897855162620544, 0.2848068177700043, 1.2307201623916626, -0.04543035104870796, -0.2700428068637848, -0.8299885988235474, -0.6530196666717529, -0.588006854057312, 1.9947856664657593, 0.6018831133842468, -0.49500033259391785, 0.5113569498062134, -0.9212366938591003, 1.74709951877594, 0.03880525752902031, -0.3276338577270508, 0.21387284994125366, 0.08713771402835846, -0.5175135731697083, 1.3878599405288696, -0.11755365878343582, -1.7335537672042847, -2.4072763919830322, 1.0378391742706299, -0.5279517769813538, -1.4019639492034912, -0.3586574196815491, -1.450730562210083, 0.9928364157676697, -1.3529075384140015, -0.8424533605575562, -0.3318667411804199, -0.04138367995619774, -0.9497488141059875, -0.4463426172733307, 0.7413652539253235, -0.305218905210495, 1.55033540725708, -0.7956405878067017, -0.9783902168273926, -0.5568780899047852, -0.7551001310348511, 1.4974547624588013, -0.5089200139045715, 0.36289677023887634, 2.291273593902588, -0.19889675080776215, 1.8823150396347046, 0.10741648823022842, -2.239358901977539, 1.0992716550827026, -0.2499980479478836, -1.4950149059295654, 0.3641676902770996, -1.5068010091781616, 1.2416983842849731, -0.24041390419006348, -0.5360628366470337, -0.3706884980201721, -1.8367446660995483, 0.35007789731025696, -0.5166739225387573, -1.0353469848632812, -0.5020286440849304, 0.33311906456947327, -0.9981285929679871, 0.12821540236473083, 0.12913969159126282, 0.08790116757154465, 0.9043717384338379, 0.3466770052909851, 0.629913866519928, -0.12854234874248505, 0.17514775693416595, 0.07632724940776825, -0.18314874172210693, -0.7551149725914001, 0.9296174049377441, -0.021696139127016068, -0.6987459659576416, 0.13159510493278503, 0.8717488050460815, 0.43553122878074646, 1.3296749591827393, -0.27884599566459656, 0.2014492005109787, -0.9226450324058533, -0.6472440958023071, 0.5731684565544128, 0.07070586830377579, -0.976266622543335, -0.22135278582572937, 0.07538515329360962, -0.45156803727149963, -0.05396757647395134, 0.4264032542705536, 0.9442668557167053, 0.8087111711502075, 0.15047436952590942, 0.9059582948684692, -1.2037361860275269, 0.6072766184806824, -0.21234478056430817, -0.5559356808662415, -0.19036787748336792, 0.46759286522865295, -1.1814539432525635, 0.5976064801216125, 0.6918986439704895, 0.18675312399864197, -1.6824320554733276, 0.5006289482116699, -0.5871534943580627, 2.598597288131714, -1.3276516199111938, -1.082413673400879, 0.2863122820854187, 0.30376508831977844, 0.4072537124156952, -0.8492522835731506, -0.33482909202575684, -1.9069997072219849, 0.2031075358390808, 0.1380019336938858, 0.9769587516784668, -0.63358473777771, 0.6105135679244995, 0.17719443142414093, 0.05936183035373688, -0.5242663025856018, -0.8654307723045349, 0.9102880954742432, -0.531425952911377, -0.24543312191963196, 1.6763813495635986, -1.514329433441162, -0.6477920413017273, -2.783719778060913, -1.012498140335083, -0.6902544498443604, -1.328397512435913, 1.3733056783676147, -1.1501505374908447, 1.45014226436615, -0.8012771606445312, 0.21981607377529144, 1.7200753688812256, 0.9164554476737976, -0.6123318076133728, -0.3774675130844116, 0.7708426713943481, -0.45109471678733826, 0.26922428607940674, 0.15797126293182373, 0.09911974519491196, 0.6001964211463928, -0.7708367705345154, 0.740029513835907, -0.5441281795501709], [0.20861080288887024, -0.49383488297462463, 0.15555228292942047, 1.7205886840820312, 1.6504682302474976, 0.832322895526886, 0.2116541713476181, -0.10895490646362305, 0.16688106954097748, 0.18848317861557007, -1.0378632545471191, -0.4985276162624359, 2.1265594959259033, -1.5193415880203247, -0.31561946868896484, 1.5316780805587769, -1.709787130355835, 1.4771674871444702, 1.5636191368103027, 1.5480986833572388, -0.4010412096977234, 0.96243816614151, 1.151212215423584, -0.35978421568870544, 0.7990103363990784, -1.523497462272644, 0.2582690119743347, -0.633608341217041, 1.4385679960250854, -1.585780143737793, -1.236481785774231, -0.8913740515708923, 1.1171584129333496, -0.18467968702316284, -2.458505868911743, 0.001240472891367972, 1.3450473546981812, -0.6352988481521606, 0.03852329030632973, -0.2804367244243622, -0.7248344421386719, 0.043043408542871475, -0.6997378468513489, 0.8062178492546082, -0.9139509201049805, 1.057799220085144, -0.17767426371574402, -0.21135778725147247, -0.055935338139534, -0.33042725920677185, -0.8661641478538513, 0.17040513455867767, 0.46201813220977783, 0.9780303239822388, -1.7874764204025269, -0.48465728759765625, -0.2701777219772339, 1.4256192445755005, 0.12173857539892197, -0.6271654963493347, 0.15553241968154907, -1.0385456085205078, -0.6567752957344055, 0.41009342670440674, 0.39035019278526306, 1.5979769229888916, -0.1671026647090912, -0.9263147711753845, 2.4943416118621826, 0.4190055727958679, 1.456838846206665, -1.1905717849731445, -0.7030474543571472, -1.6190471649169922, 0.9404088854789734, 1.5726940631866455, 2.0054190158843994, 1.983473539352417, 0.8733632564544678, 0.2467988133430481, -0.24855011701583862, 1.0430841445922852, 0.30191731452941895, -0.5760899186134338, -0.35914328694343567, 1.2890665531158447, 1.1650910377502441, 1.0509003400802612, 0.36564862728118896, 1.3069379329681396, 0.8543632626533508, 0.10803449153900146, -1.7637444734573364, 0.24398218095302582, 0.589479386806488, 0.20288759469985962, 1.590705156326294, -0.03612305223941803, 1.5121873617172241, 0.5290383696556091, -0.3229193687438965, 0.18923549354076385, -0.983026921749115, -0.2409341037273407, 0.14913249015808105, 0.2383401095867157, -0.37040817737579346, 0.9187130928039551, 0.02992461994290352, -2.0597329139709473, 0.4981635808944702, 0.3386584520339966, -0.46405860781669617, -2.577016592025757, 0.42941486835479736, 1.6484768390655518, 1.8736141920089722, 0.23188060522079468, 1.770484447479248, 0.541164219379425, 0.5285136103630066, 1.1232534646987915, 0.4389306902885437, -1.1525781154632568, 0.3520802855491638, 0.0989832729101181, -0.32807740569114685, 1.1888923645019531, 0.816882312297821, 0.9718561768531799, -1.3619496822357178, -2.0280508995056152, -0.5346646308898926, 1.454369306564331, -1.3610483407974243, -0.760487973690033, -1.8052012920379639, -0.7790311574935913, 1.2430201768875122, -0.5198538303375244, 0.5501474142074585, -1.059173345565796, 1.586077332496643, -0.12678919732570648, -1.7171566486358643, 0.8822967410087585, 0.7447549104690552, 1.124511480331421, 0.1329004168510437, -0.3981275260448456, -0.7367169260978699, -0.2590583860874176, -0.3036033511161804, -1.626994013786316, 0.8030040264129639, 1.075208067893982, -0.24934908747673035, 0.18638332188129425, -0.43184664845466614, 0.8614494800567627, 1.5199100971221924, -0.3735545873641968, 1.2506022453308105, -0.9564437866210938, 1.1769696474075317, 0.3381024897098541, -0.3414371609687805, -0.0607282891869545, -1.306793212890625, -0.4970957934856415, -1.0829319953918457, -0.35196009278297424, -1.0794514417648315, -0.8643026947975159, -1.2810771465301514, -0.4226197898387909, 1.3979676961898804, 0.6478862166404724, -0.5847692489624023, 0.8821614980697632, -0.3141058385372162, -0.6973837614059448, -0.013805116526782513, -0.5851395130157471, -1.1943672895431519, -0.19856616854667664, -0.48254671692848206, 0.6626181602478027, -0.19693566858768463, 0.6200703382492065, 0.2506864368915558, -1.271618127822876, -0.6100165843963623, 0.07033931463956833, 0.9328725337982178, 0.7169347405433655, -0.7673846483230591, -0.8575635552406311, 0.7895691394805908, -0.5908493399620056, 0.7518206238746643, 0.42503559589385986, 0.7786632180213928, 0.05257841572165489, -1.8824976682662964, -1.6914197206497192, -1.3964890241622925, 1.762953758239746, -0.020117878913879395, 0.2969706058502197, 1.095442295074463, 0.49825695157051086, -0.5889109969139099, -0.6183868050575256, -0.285978764295578, 0.4612622857093811, -1.4702577590942383, -0.43534234166145325, 1.2846488952636719, -0.3872551918029785, 2.141031503677368, 0.21175526082515717, 0.07197123020887375, 0.23355476558208466, -0.2905716896057129, 0.06493572145700455, 1.1711257696151733, -0.20642618834972382, 1.0676382780075073, 0.583672821521759, 0.24628543853759766, 0.09797736257314682, 0.3974417448043823, -0.9385653138160706, 1.4324132204055786, -0.526606023311615, -0.3151758015155792, -0.0005181230371817946, -1.4622853994369507, 0.7947741746902466, 0.47660550475120544, -0.3334566652774811, 1.2147448062896729, 0.06384942680597305, 0.737616240978241, -0.9006766080856323, -0.4303511679172516, 0.044843640178442, 0.4848284125328064, 2.1145167350769043, 0.13137876987457275, -0.38556045293807983, -1.5178455114364624, 0.6575413942337036, 0.7441134452819824, 1.0735855102539062, 1.0649818181991577, 1.2335360050201416, -0.38350042700767517, -0.18142475187778473, -0.32491862773895264, -1.9413036108016968, -0.9034823775291443, 0.5464970469474792, -0.8157743811607361, -1.1255844831466675, 0.13366103172302246, 0.15604746341705322, 0.36057305335998535, 0.3281272053718567, 0.863535463809967, 1.1973505020141602, 1.3795762062072754, -0.5756336450576782, -0.7369574904441833, 1.053386926651001, 0.25366368889808655, 0.6633502840995789, -1.1282424926757812, -0.8490099310874939, 0.26550811529159546, 0.2023683786392212, 0.832761824131012, -0.2926872670650482, -0.5186973214149475, 0.07389658689498901, -0.5455955266952515, 1.324045181274414, -1.734177589416504, 1.5100184679031372, -1.1450086832046509, -0.8827912211418152, 1.7952157258987427, -0.7227802872657776, 0.8739351034164429, 0.016632085666060448, -1.2894121408462524, 0.6600084900856018, -0.6787216067314148, -0.054187092930078506, 0.7941970229148865, -0.7531505227088928, -1.0346617698669434, -0.30325788259506226, -0.039162855595350266, 1.5257490873336792, -0.08710407465696335, 0.08632177114486694, 1.810821294784546, -1.6620094776153564, 2.4779434204101562, -0.22977161407470703, 0.608548104763031, -0.28797051310539246, 1.0858038663864136, -0.5463602542877197, 1.669853925704956, -0.7143338918685913, 0.3158503472805023, -0.23311707377433777, 0.1605915129184723, 0.14903044700622559, 0.49343082308769226, 0.5966933369636536, 0.14236313104629517, 0.4096564054489136, 0.36885204911231995, 2.055290937423706, 0.5111269950866699, -0.8443934321403503, 0.6083921790122986, 0.014986694790422916, -1.3056094646453857, -1.5474684238433838, -0.9252676963806152, -0.3937057554721832, -0.7837162613868713, 0.29522705078125, -1.2039424180984497, 0.5126011371612549, 0.01164951827377081, -0.6654465794563293, 1.659746766090393, 0.9463540315628052, -0.8600725531578064, 1.2716972827911377, -0.6961632370948792, -2.168825626373291, 0.33450087904930115, 0.10437322407960892, -0.16219106316566467, 0.42153194546699524, -0.4198380410671234, -0.16301903128623962, -1.0226856470108032, 0.323930561542511, 0.22145040333271027, 0.3345130980014801, -0.29335951805114746, 0.6327323317527771, 0.5719625353813171, 1.6262171268463135, -0.9469426870346069, -0.4367060661315918, -1.3400527238845825, 1.1012139320373535, 0.2625778317451477, 0.9263762831687927, -0.334602028131485, 0.3729182779788971, -0.3185659348964691, -0.5884581208229065, 1.0789562463760376, -0.6938378810882568, 1.3457920551300049, 0.028422491624951363, 0.03999721631407738, -1.902395248413086, -0.5205777883529663, -0.3593578040599823, 0.03609443083405495, -1.0585640668869019, 0.942757785320282, -0.286965548992157, -0.7970548868179321, -1.1990628242492676, -1.3941869735717773, -0.6358895897865295, -0.4676983952522278, -0.6192793846130371, -0.057011257857084274, 1.1604334115982056, -1.401845097541809, 0.3116927444934845, -0.005635197274386883, 0.5370562076568604, -1.4110612869262695, 0.37854284048080444, -0.11141543090343475, 0.8471429347991943, -0.2707311511039734, -1.074890375137329, -1.4833687543869019, -0.3930637538433075, -0.9278997182846069, -0.07750356197357178, 1.4129514694213867, 0.90186607837677, 1.67430579662323, -1.132145643234253, 1.153639316558838, 0.21134407818317413, -2.4793169498443604, -0.5996428728103638, -1.1723780632019043, 0.6528134346008301, -0.3637492060661316, 1.3941928148269653, 1.9893804788589478, -0.923453688621521, 0.5570066571235657, -0.49553462862968445, -0.8543871641159058, 0.7256642580032349, 0.5249395370483398, 2.074685573577881, -1.5481542348861694, 1.9157496690750122, -0.5982736945152283, -0.1797313541173935, 0.5131819844245911, 0.2804005444049835, -0.11074782907962799, 0.5266439318656921, 1.3387888669967651, 0.5092945098876953, 0.859149694442749, 0.6169799566268921, -0.8021413087844849, 0.0675911009311676, 0.958595335483551, 0.7107168436050415, -1.8712992668151855, 0.08776339143514633, 0.3901446759700775, 0.4643937647342682, 0.9481120109558105, -0.7413297295570374, 0.2508225739002228, 0.3792918622493744, 0.1658158153295517, 2.679297924041748, -0.09408450126647949, 0.3246789574623108, 0.8752813339233398, 0.4069230556488037, -1.215632438659668, 0.773398220539093, 0.8983489871025085, 1.8542985916137695, 0.43172240257263184, 1.177227258682251, 0.4741821587085724, -1.3333687782287598, 0.16228239238262177, -0.47528496384620667, -0.08816593885421753, 0.0367896668612957, -0.13933321833610535, -0.41537144780158997, 0.7242229580879211, -1.0883852243423462, -2.562182664871216, -0.5885853171348572, -1.7128632068634033, 1.8840333223342896, -0.9204055070877075, 1.2150545120239258, 1.3845813274383545, 0.9399756789207458, 0.05454403534531593, 0.3957880437374115, 0.859049379825592, 0.2150591015815735, -0.39025452733039856, 0.4395596385002136, 0.7352467775344849, -0.060479115694761276, -0.8926948308944702, 0.013648951426148415, 0.04065597429871559, -0.026024561375379562, -0.7438085675239563, 0.06286006420850754, 0.5345861911773682, 1.3397433757781982, 0.22618111968040466, 1.1065529584884644, -2.2465875148773193, -1.1094508171081543, -2.323392868041992, 0.7655788660049438, 0.1373901069164276, 1.4259735345840454, 0.023598723113536835, 0.546429455280304, 0.4305238723754883, 0.3837917745113373, 1.6602064371109009, -1.3441267013549805, -1.8961080312728882, -0.6556531190872192, 0.11357728391885757, -0.8849868774414062, 2.062288284301758, 0.6817610263824463, -1.3109898567199707, -0.809317409992218, -0.8290873765945435, -3.6625452041625977, 1.003731608390808, 0.5495386123657227, -2.1484837532043457, 0.5945752263069153, 0.27436310052871704, -0.8631905913352966, -0.16303175687789917, -0.8594541549682617, 0.0065132793970406055, 0.00045687390957027674, -0.5881648659706116, -0.0386713482439518, -0.065989188849926, -2.0624091625213623, -0.9821524620056152, 0.9424541592597961, 0.898758590221405, 1.1395572423934937, 0.8116224408149719, 0.4807916283607483, 1.0473897457122803, 0.3577263653278351, 0.05163862928748131, -0.5465542078018188, 0.3502553403377533, -0.678846538066864, -0.8327667117118835, 0.6962996125221252, -3.5487544536590576, 0.7673793435096741, -2.820507287979126, -0.18477822840213776, -0.2616857588291168, 1.0331510305404663, 1.4562976360321045, 1.1504133939743042, -0.47220578789711, 0.08181619644165039, -0.05576983839273453, -0.5372245907783508, -1.2209525108337402, -0.23578089475631714, -0.08598372340202332, 1.3963418006896973, -0.27994415163993835, -0.8742642998695374, -1.5243874788284302, 0.024964572861790657, 1.0990431308746338, 0.013578207232058048, -0.5465838313102722, -0.8881669640541077, -0.599564790725708, -0.2122814655303955, -0.807757556438446, 0.7568308115005493, -0.5782802700996399, -0.4725385904312134, -0.021077098324894905, 0.29186105728149414, -0.17769674956798553, -0.20721805095672607, -1.3813291788101196, 0.20868156850337982, 2.281367301940918, -0.006577524822205305, 0.7757910490036011, -1.1249395608901978, -0.9054029583930969, -0.6460573673248291, 0.9252846240997314, 0.4252408742904663, -0.6318259239196777, 0.014562376774847507, 0.164912149310112, 0.30471235513687134, 0.11870770156383514, -0.13173159956932068, 0.747236967086792, 1.3300578594207764, -1.5015367269515991, -0.0013473369181156158, -0.16155560314655304, -0.527249813079834, 1.6767394542694092, -0.5046220421791077, -0.2685357630252838, -0.8169019222259521, -1.7468072175979614, -2.0776169300079346, -0.19622330367565155, -0.7452052235603333, 1.7210161685943604, 0.31646037101745605, -0.7368480563163757, 0.5731101632118225, -0.14838917553424835, 2.2026331424713135, -1.4645215272903442, 0.13696980476379395, 1.2899576425552368, 0.2866832911968231, 1.0873870849609375, 0.401131272315979, 0.23027293384075165, -0.16124559938907623, 1.2682791948318481, -0.9926849603652954, 0.11520669609308243, -0.4413389563560486, -1.2834813594818115, 0.6471357345581055, 0.6421877145767212, 0.5062286853790283, 0.6949519515037537, -1.6273882389068604, -0.7395216226577759, -0.21185167133808136, -0.5625289082527161, -1.6087241172790527, 0.30292239785194397, 0.4632873833179474, -1.5855600833892822, 0.6125035881996155, 1.1062613725662231, -0.8286573886871338, -1.1927093267440796, 0.5672149062156677, -0.6716611981391907, 5.94443226873409e-05, 1.8987373113632202, -0.49930092692375183, -1.1591429710388184, 1.3696370124816895, 0.7380045056343079, -0.0316903293132782, 0.257163941860199, -0.7142587900161743, 0.1523064523935318, -0.8670189380645752, -0.23095780611038208, 0.9969484806060791, -0.25854215025901794, -0.25776341557502747, 0.23194527626037598, -1.291530728340149, -0.023662829771637917, 0.21539416909217834, -3.5636205673217773, 0.08059120178222656, 0.5818367004394531, 1.395725965499878, -1.0345145463943481, -0.3128508925437927, -0.11622916907072067, -0.26879236102104187, 0.9591624140739441, -1.2228506803512573, -0.5324546694755554, 0.11264045536518097, 0.9837393164634705, 0.19914397597312927, -1.5214965343475342, -1.2722220420837402, -0.4894102215766907, 0.9560878276824951, 0.7646239399909973, -0.5214152932167053, 1.0150140523910522, -0.9385588765144348, 0.919367253780365, -0.022614281624555588, -0.13985514640808105, 0.3536185622215271, -0.3870355486869812, 0.07158584147691727, -0.3071814477443695, -0.13901421427726746, 0.5805066227912903, -0.29692918062210083, 1.485226035118103, -1.0967302322387695, 0.16289293766021729, -0.1278766244649887, -0.04656635597348213, 0.6804869174957275, -1.076812505722046, 0.23701907694339752, 0.9130333065986633, -0.4023755192756653, 0.18523210287094116, -1.7351266145706177, 1.9509525299072266, -0.36359089612960815, -1.030820608139038, -1.359197735786438, -0.6369243860244751, 0.42433181405067444, 1.6836475133895874, -1.1052085161209106, 1.6982309818267822, -0.2892911732196808, 1.5446507930755615, -0.803826093673706, 0.20280291140079498, 1.5582128763198853, -2.2353017330169678, 1.2748874425888062, 0.08528445661067963, -1.9740791320800781, 0.027120312675833702, 0.06665828824043274, -0.8086944222450256, -0.9401105642318726, 0.9205164909362793, -0.8873333930969238, 0.5911656618118286, -0.1464049220085144, 0.23382343351840973, -0.4126894176006317, -0.5678531527519226, -0.6185689568519592, -0.3080480992794037, 0.26579737663269043, -1.6284712553024292, 0.04953581094741821, 0.31014496088027954, 1.6222925186157227, -2.6786770820617676, -1.4110994338989258, 1.0969223976135254, 0.5774646997451782, 0.8269504308700562, -0.7194200158119202, -0.5047100186347961, -0.5651615262031555, 1.0275278091430664, 0.5724449753761292, 0.035606976598501205, -0.07117439806461334, -2.1333882808685303, -1.0925601720809937, 0.3999398946762085, -0.47860825061798096, -0.004996843170374632, 1.517046332359314, -0.48322007060050964, -1.1637225151062012, -0.06195995211601257, -0.49143046140670776, 1.0985559225082397, -2.2020881175994873, 0.5948516130447388, -1.2016562223434448, -2.374451160430908, 0.8477492332458496, -0.02103489078581333, 1.5912866592407227, -0.44826751947402954, -0.48691049218177795, 1.4369477033615112, -2.0001938343048096, 1.124058723449707, -0.08038533478975296, -0.9170058369636536, 0.8636391162872314, -1.5084604024887085, 0.1406061351299286, 0.8344069123268127, -1.81344735622406, -0.5580700635910034, -1.2938268184661865, 0.08382821083068848, 0.8107859492301941, 0.40244871377944946, 0.07694372534751892, 0.685775876045227, 1.5844519138336182, 0.3835958242416382, 0.8974915146827698, 0.3945165276527405, 1.6982264518737793, 0.26419079303741455, -1.359757900238037, 0.19028250873088837, 0.5398639440536499, -0.36226174235343933, -1.2019243240356445, 0.84906005859375, 0.14536818861961365, -0.03399252891540527, 0.6123707890510559, -0.7805939316749573, 0.691992998123169, 0.9238131046295166, -0.5705699920654297, 0.7530926465988159, 1.0285195112228394, -0.7528820037841797, 0.24013328552246094, -0.013513138517737389, -0.14059774577617645, -0.8998719453811646, -1.7146958112716675, -0.8183476328849792, -0.32462796568870544, 1.1309740543365479, -0.15710656344890594, -0.041004106402397156, -0.8068872094154358, 0.3256627321243286, -0.7059298753738403, -0.28772851824760437, 1.0525684356689453, -0.35635748505592346, 0.5211354494094849, -0.28815287351608276, 0.2783077657222748, 0.5684162378311157, -2.289764404296875, 0.7626933455467224, 0.7661567330360413, 0.22744305431842804, -1.3051701784133911, -0.8363913893699646, -0.8901764154434204, -0.8814095258712769, -1.2049516439437866, -0.44206804037094116, 1.3337055444717407, -0.47504931688308716, 1.4119682312011719, 1.2122174501419067, -1.4879653453826904, 0.42986392974853516, 0.07448329031467438, -0.17424213886260986, 1.0366346836090088, 0.30441269278526306, -1.3743640184402466, 2.0428993701934814, 0.7993902564048767, -0.6951230764389038, 0.8614079356193542, -0.11326329410076141, 1.142470359802246, -0.9221317172050476, -0.7166930437088013, 1.4662877321243286, 0.2526627480983734, -0.8866241574287415, 1.1528866291046143, 0.4482593536376953, 0.3464375138282776, -0.16664238274097443, 0.923134446144104, 1.4524872303009033, 0.8491367101669312, 0.6721198558807373, -0.1135801449418068, -0.3472480773925781, 1.3262615203857422, 1.0554907321929932, 0.2584353983402252, -1.4546996355056763, 0.17187832295894623, 0.5152791142463684, -0.41055187582969666, 0.2131717950105667, 0.33218061923980713, 0.14656023681163788, 1.9528062343597412, 0.304685115814209, 0.7959274649620056, -1.5450454950332642, -1.0036824941635132, -0.5925859212875366, 0.8381292223930359, -2.586287498474121, -0.7160177230834961, -0.18541888892650604, 0.05527808517217636, 0.4484506845474243, 1.1534699201583862, -0.15364892780780792, -0.09875188767910004, 0.3181360065937042, 0.10060840845108032, 0.5045751929283142, -0.7831193804740906, -0.8672229647636414, -0.7705934047698975, -1.5038048028945923, 1.3841907978057861, -0.9895416498184204, -0.1467854380607605, -0.045746587216854095, -0.3229478597640991, -0.8004739880561829, 1.242855429649353, -0.5135939121246338, 0.5486389994621277, 0.17582884430885315, 0.4533711373806, 0.389360249042511, -0.9817078709602356, -1.618144154548645, 1.193938136100769, 0.7420650124549866, -0.18697616457939148, 0.10095179826021194, 0.9301125407218933, 0.5183017253875732, -1.540583610534668, -0.09985218197107315, -1.1330026388168335, -1.9095677137374878, 1.9760934114456177, 1.187841534614563, -0.24217940866947174, -0.0884971022605896, -1.692643165588379, 0.7884197235107422, 1.9562543630599976, -0.3293015956878662, -0.16570648550987244, -1.817211389541626, -0.9869958758354187, 0.7657368779182434, -1.0122462511062622, -0.41179394721984863, 0.7981690764427185, -0.1379460096359253, -0.9128130078315735, -0.5835244059562683, -0.5509675145149231, 1.4027841091156006, -2.0504252910614014, 0.5534892082214355, 0.1418694257736206, -0.20111481845378876, -1.008265733718872, 0.6886292099952698, -2.2378249168395996, 1.3452935218811035, -1.2724905014038086, -0.6927732229232788, 0.630355179309845, 0.05810756981372833, 0.22048266232013702, 1.2149386405944824, -0.5774869322776794, -0.45615679025650024, -1.3343987464904785, -0.707193911075592, 1.6130003929138184, 0.10536239296197891, 0.9698489904403687, -0.39769983291625977, -1.3354557752609253, -0.031870175153017044, 0.2907334566116333, 0.8691085577011108, 0.7101143598556519, 0.28329169750213623, -1.3472115993499756, -0.30344220995903015, -1.0669583082199097, 0.536088764667511, 1.0880290269851685, -0.1418323516845703, 0.7222165465354919, -0.057423800230026245, -0.4613629877567291, -1.431313395500183, 1.0623830556869507, -0.9493920803070068, -2.045316457748413, -1.7724401950836182, 0.3845864534378052, 1.4956012964248657, 0.27643021941185, 1.9113410711288452, -0.6091674566268921, 0.949671745300293, -2.61892032623291, 1.0587844848632812, 1.8597617149353027, 0.10515344142913818, 0.6056388020515442, -0.632342517375946, 1.3234531879425049, 0.6528574824333191, -0.6507986783981323, -2.5760865211486816, -0.20664668083190918, 1.1231412887573242, 0.6234673857688904, 0.711685061454773, -0.2484883815050125, 0.6680045127868652, -2.016152858734131, -1.0466810464859009, -0.11347207427024841], [-0.5929555296897888, -0.9457987546920776, -1.5383833646774292, -1.3025656938552856, 0.6970244646072388, -1.9472458362579346, 0.16242271661758423, -0.48214638233184814, 1.003212809562683, 0.33112701773643494, -0.3605150282382965, -2.051914691925049, -0.4568190574645996, -0.21554780006408691, 0.16353604197502136, -0.36963123083114624, -0.2700650691986084, -0.1486491858959198, -0.2528531551361084, 0.45820343494415283, 0.8418828845024109, -0.19126221537590027, -0.6160218715667725, 0.44247859716415405, -0.15253306925296783, 0.3664530813694, 0.0876913070678711, 0.8185855150222778, 0.2794286906719208, 0.12623976171016693, -1.3905119895935059, 2.1067919731140137, 0.2146930992603302, -0.8327828049659729, -0.5701067447662354, -1.102057695388794, 0.40113365650177, -1.215478777885437, -0.7251896262168884, 1.2791054248809814, 0.42367395758628845, -0.13287803530693054, 1.2590718269348145, -0.8156278729438782, -0.15822823345661163, 0.18928959965705872, 0.3138071596622467, -2.4584105014801025, 0.8779767751693726, 0.6325390338897705, -1.1729481220245361, -1.8702808618545532, 1.2453265190124512, 0.20879057049751282, 1.4395672082901, 0.9786741733551025, 1.20072340965271, 0.119622141122818, 0.7948752641677856, -1.942832350730896, 0.6851968169212341, 0.9440303444862366, 0.26892173290252686, 0.6503961086273193, 0.4711958169937134, 0.47275757789611816, -1.5042811632156372, -0.08389151841402054, 0.45573219656944275, 1.5031623840332031, 0.12775105237960815, -0.06170303747057915, 1.0565649271011353, 1.092478632926941, -0.550370454788208, 1.0060858726501465, 2.3277885913848877, -1.1164534091949463, 0.33804380893707275, 0.5434809923171997, -1.3596495389938354, 0.32267117500305176, -0.28668659925460815, -1.7802541255950928, 1.5526173114776611, 2.0148303508758545, 1.1357295513153076, 1.2179508209228516, 0.3031460642814636, -0.06395978480577469, -0.45253774523735046, -0.9830114841461182, -2.5169568061828613, 0.33997759222984314, -3.0095419883728027, 0.023829998448491096, -0.5327159762382507, 0.11764488369226456, 0.5931634902954102, 1.754880428314209, 0.27867430448532104, 1.1001523733139038, 0.7891896963119507, 0.18069568276405334, -0.066557377576828, -2.4196577072143555, 0.7447375059127808, -0.6318404674530029, 0.2945782542228699, 0.37908294796943665, 0.1328052431344986, -1.0334891080856323, -1.905595302581787, -0.17037278413772583, 1.5169813632965088, -1.08196222782135, -0.5194189548492432, 0.37027743458747864, -0.7834988832473755, 1.1353075504302979, -1.5936452150344849, 0.4703925549983978, -0.6444743871688843, -2.32607364654541, -1.1237757205963135, -0.8991418480873108, 0.600818932056427, -0.2435869723558426, 0.1697404533624649, -0.042479369789361954, 0.522502601146698, -1.1681290864944458, -1.387894630432129, 0.014101685024797916, 0.25123104453086853, -0.7635661363601685, 0.15462979674339294, -1.8491989374160767, -1.8779667615890503, 1.1061512231826782, 1.265371322631836, 0.08497558534145355, 0.3421765863895416, 0.3831591308116913, -0.45938074588775635, -0.032187171280384064, 0.7031720280647278, 0.7694658041000366, 0.21826042234897614, 0.9712604284286499, 0.8715533018112183, -1.3010072708129883, -0.4031795859336853, 0.013041280210018158, -1.7523421049118042, -0.1826004981994629, 1.0803117752075195, 1.3764638900756836, -1.1149210929870605, 0.5504985451698303, 0.8508599996566772, 1.0054253339767456, -0.3934940695762634, -0.3315357565879822, 0.5342742204666138, -0.48375388979911804, -0.39212802052497864, -0.3257448673248291, 2.8697595596313477, -0.2871858477592468, -1.4110851287841797, -0.35787540674209595, -0.5694516897201538, 0.30126819014549255, 0.8972386121749878, 0.3247256278991699, -0.8369231224060059, 0.143232062458992, -0.7714080214500427, -1.0210925340652466, 0.06547701358795166, -1.1214852333068848, 0.304775208234787, -1.139437198638916, 0.648790717124939, -0.8297394514083862, 0.39119571447372437, 0.7580180168151855, -1.149592638015747, 0.5503106713294983, -1.1175342798233032, 1.3613200187683105, -0.3947390615940094, -0.6705586910247803, -1.3970941305160522, 0.1296236217021942, 0.6210245490074158, 1.566834807395935, 0.935201108455658, -0.3976760804653168, -0.30063092708587646, 0.6565405130386353, 0.047709401696920395, 0.27797141671180725, 1.5314916372299194, -0.2210502028465271, 0.392254501581192, 0.7748141288757324, -0.9566219449043274, 0.3606325387954712, -0.40407511591911316, -0.12790603935718536, 0.48417434096336365, 0.7175601720809937, -0.4412150979042053, -0.9289896488189697, -1.7328931093215942, -0.720278263092041, -1.1883280277252197, -1.1702841520309448, 1.9560691118240356, 0.0064318072982132435, 0.44045910239219666, -0.09161059558391571, -0.9099528193473816, 0.049815572798252106, -0.3684367537498474, -0.7710934281349182, -0.6413002014160156, 1.0613737106323242, 1.7855663299560547, 1.560936450958252, -1.0006059408187866, -0.02568002976477146, 0.43033313751220703, -0.8882716298103333, -0.5018116235733032, -0.3088182806968689, -0.6934524774551392, -0.877117931842804, 1.041203260421753, -0.4139153063297272, -0.7820538878440857, 1.0841243267059326, -1.187705397605896, -1.5377687215805054, -1.23356294631958, -0.5502023100852966, -1.4427419900894165, -1.3548825979232788, -2.035585641860962, 0.19307750463485718, 1.202987790107727, 0.5105028748512268, -0.42675796151161194, 0.7461951375007629, -0.41661229729652405, 0.024229295551776886, 0.21179227530956268, -1.3076741695404053, 0.9216777086257935, 0.4501950740814209, 0.0015095035778358579, -0.048240941017866135, -0.1997174322605133, 0.1318051666021347, -1.4767792224884033, -1.5889323949813843, 0.2739708125591278, -0.6155694723129272, -0.8676927089691162, -0.06910018622875214, 0.21146687865257263, -0.1738179624080658, -0.25783777236938477, 1.6672159433364868, 0.5265622735023499, 0.04543904960155487, -0.5021135807037354, -0.7231000065803528, 0.14257307350635529, -0.20897923409938812, -1.441023588180542, -0.20050720870494843, 0.5657968521118164, -0.6931535005569458, -1.764532208442688, 0.11120329797267914, -1.152787685394287, -0.0613604299724102, -0.5215557813644409, 0.29569852352142334, 1.528731107711792, 0.5038967728614807, 0.41197726130485535, 1.080735206604004, -0.5728614330291748, 0.9985653162002563, 1.0797736644744873, -0.36837267875671387, 0.518834114074707, -0.05538436397910118, -0.47557827830314636, -1.1311784982681274, 1.567470908164978, -1.2759774923324585, 1.4801050424575806, -0.2279239147901535, -0.060443565249443054, 0.11863842606544495, 0.24381023645401, 1.7498077154159546, -1.6177144050598145, -0.09337181597948074, -0.7263519763946533, 0.16110748052597046, 0.19349788129329681, 0.5572367310523987, -0.37480324506759644, 0.4889522194862366, 1.7530711889266968, 2.2017359733581543, 1.1136596202850342, -0.8584809303283691, -1.3707542419433594, 0.3918374478816986, -1.6227062940597534, 0.06333453208208084, -0.1253155916929245, 1.6515071392059326, -0.44756609201431274, -0.5563789010047913, 0.8973219394683838, 1.4814587831497192, -0.3984738290309906, -1.2824255228042603, -0.47939181327819824, -0.6401779055595398, -0.19700956344604492, 0.8470956683158875, 0.9131078720092773, 0.441186785697937, 0.2398153841495514, 1.607949137687683, 0.19924664497375488, 1.2585325241088867, -1.94666588306427, -0.9543454051017761, -1.8519384860992432, 0.5200265049934387, -0.5147411227226257, 1.0627745389938354, 1.7056928873062134, 0.04004678875207901, -1.0954923629760742, 0.4512408971786499, 0.08625638484954834, -0.806204617023468, 2.264991521835327, -0.6375378966331482, -0.6421003937721252, -0.1660730540752411, 0.36582985520362854, -0.7163262963294983, -0.4166056513786316, -1.4292374849319458, -2.3153293132781982, -0.8488025665283203, -0.6889532804489136, 0.7240586876869202, -0.47837260365486145, -0.03966154903173447, -1.5604726076126099, -2.246868371963501, -0.3798530399799347, 0.6327566504478455, -0.21319471299648285, 0.1818675547838211, -1.3029991388320923, 0.047277215868234634, -0.7554475665092468, 0.8253549337387085, -0.1551303118467331, -0.3071792423725128, 1.0125501155853271, -0.13580794632434845, -0.10157288610935211, 0.5221506953239441, 1.0511542558670044, 0.856743335723877, 1.1605076789855957, -0.3510703146457672, 0.43962064385414124, -0.08031313121318817, 0.4251486659049988, -0.5982110500335693, -0.22124741971492767, 0.4244632124900818, 1.1790119409561157, -0.24882754683494568, -0.8242865800857544, 0.10571915656328201, 0.16481463611125946, -0.5488706231117249, 1.6278786659240723, 0.4293505847454071, 1.510649561882019, -0.029449917376041412, -0.1198490634560585, -0.13491618633270264, 0.5459314584732056, 1.4089629650115967, 0.13858261704444885, 0.08276969194412231, -0.8347647786140442, -1.816265344619751, 0.44465556740760803, 1.1550943851470947, -0.15372496843338013, 0.20690472424030304, -0.18722516298294067, -0.2752608060836792, 0.49522173404693604, -0.2895704507827759, -1.3017160892486572, 0.6552833914756775, 0.8309292197227478, -2.6395506858825684, 1.3252977132797241, 0.6313611268997192, 1.4156385660171509, 0.2322075366973877, -1.1302423477172852, 0.24311745166778564, 0.8282966017723083, 0.5862451195716858, -0.22760123014450073, 0.5843192934989929, -1.1093535423278809, 0.5474614500999451, -0.005720267537981272, 0.33946168422698975, -0.18607056140899658, -0.17758868634700775, 0.2425241470336914, 1.4497698545455933, -0.17523100972175598, 0.15086345374584198, -1.1473087072372437, -1.4011491537094116, -0.7404862642288208, -1.3834813833236694, -0.7015568017959595, 0.8229023814201355, 0.21025946736335754, -0.4229655861854553, -1.0493561029434204, 0.3975023627281189, 0.8303715586662292, 1.115558385848999, 0.5287446975708008, -0.2864198684692383, -0.6729852557182312, 1.208292007446289, 0.07702663540840149, 0.21582907438278198, -0.9043892621994019, -0.21337993443012238, -0.2307777851819992, 0.0313817523419857, 0.44324415922164917, 0.4522075653076172, -0.5844043493270874, -0.5072387456893921, -0.6624054908752441, 1.8620105981826782, -1.7075762748718262, 1.243866205215454, 0.619835376739502, -0.14914068579673767, 0.4892388582229614, 0.9673852920532227, 1.1436641216278076, -0.08029118180274963, 0.3245174288749695, 1.7548274993896484, -2.0559940338134766, -1.2488112449645996, 1.960750937461853, -0.972289502620697, 0.8010556697845459, 0.6113226413726807, 0.9875836372375488, -0.00395087618380785, 0.7139775156974792, -0.6085891127586365, 0.739891529083252, -0.41133981943130493, -0.1497505158185959, -0.6414403319358826, -0.027517708018422127, -1.6825251579284668, 0.6167764663696289, -0.13085612654685974, 0.26235127449035645, -0.5061731934547424, -0.2555944323539734, 0.5596861839294434, -0.795504629611969, 0.9802515506744385, -0.11713237315416336, 0.6995538473129272, -0.6731925010681152, -0.01671481691300869, 1.7951617240905762, 0.9348304867744446, 0.2079399675130844, 0.8003069162368774, -0.7563523054122925, -1.3363455533981323, 1.2498127222061157, -0.35119977593421936, -1.3708207607269287, -0.7594656348228455, 1.0367704629898071, 1.797029733657837, -0.06125296652317047, 0.7715815901756287, -0.5958223938941956, 0.15754373371601105, 2.049154043197632, -1.294643759727478, 0.985454797744751, 0.17904718220233917, -0.8662101626396179, 1.1242280006408691, 0.4528654217720032, -1.06374192237854, -2.2920544147491455, -0.8482945561408997, 0.46561530232429504, -1.4202311038970947, 1.552309274673462, 0.9904180765151978, 1.2960187196731567, -1.5914934873580933, 0.0345999076962471, -0.01626267470419407, 0.15357555449008942, -0.09785181283950806, 1.4131771326065063, -0.8005512952804565, 0.053172387182712555, -0.9940953254699707, 0.7367011904716492, -0.8480032086372375, -1.9512946605682373, 0.8039404153823853, 0.9100624322891235, -0.9277915954589844, -0.6379708051681519, -0.883413553237915, -0.8917524218559265, -0.9153412580490112, -0.3462340533733368, 0.2239055186510086, -0.6785426735877991, -0.26972830295562744, 0.2172204554080963, -0.7616260647773743, -0.5054832100868225, 1.667620301246643, 0.7932571172714233, -0.9563113451004028, -0.8388404846191406, 0.5160838961601257, 0.15578600764274597, -0.29565542936325073, 1.3502918481826782, 0.15693891048431396, -0.33940592408180237, 0.29169952869415283, 1.720625877380371, -0.6584244966506958, 0.514674723148346, 0.543894350528717, -0.9672728776931763, -1.2190216779708862, -1.5164148807525635, -1.2140388488769531, 0.05517907068133354, -0.18608519434928894, -1.272266149520874, -2.243549108505249, -1.3319767713546753, 0.9229158759117126, 0.8303083777427673, 0.11998293548822403, 1.0511435270309448, 1.427186369895935, 1.2292841672897339, 0.8262554407119751, 0.24287691712379456, 0.4674554467201233, 0.8817340135574341, 0.21049067378044128, -1.3479801416397095, 0.27483856678009033, -0.20385225117206573, 0.40438443422317505, 3.0175485610961914, -1.7053009271621704, 1.1778572797775269, 0.3400256931781769, 1.1715292930603027, -0.7308388948440552, -0.740664541721344, -1.6983263492584229, 0.8738051652908325, -0.446032851934433, 0.2569345533847809, -2.375424385070801, -0.5457978844642639, -1.452225923538208, 1.0803942680358887, -0.2608951926231384, 0.736914873123169, -0.1259906142950058, -0.013412480242550373, -0.6292216777801514, -0.3246634304523468, 0.1299203783273697, 0.10718755424022675, 1.457210898399353, 0.498636394739151, 0.7007072567939758, 0.39661216735839844, 0.2162034809589386, -0.6083498001098633, -0.38396015763282776, -0.9820414781570435, -0.17504501342773438, 0.22932015359401703, -1.922271966934204, 0.8766181468963623, -0.6778209209442139, -0.07931938022375107, -0.1084689274430275, 1.0163359642028809, -1.153778076171875, 0.6884158253669739, 2.0920143127441406, -0.08489607274532318, 0.032342687249183655, -1.3424768447875977, -0.1363992840051651, 1.1849948167800903, 0.27816206216812134, -0.8860823512077332, 2.2960927486419678, -1.3966710567474365, -0.5911082029342651, 0.3422154486179352, -0.38417553901672363, -1.2574034929275513, 0.05817499756813049, 0.5951647162437439, 1.6927305459976196, 1.6058636903762817, 0.08567636460065842, -0.965779185295105, 0.1642865538597107, -0.7919024229049683, 0.24606481194496155, -0.5656043887138367, -1.5239375829696655, -1.0147229433059692, 0.7918069958686829, 1.125798225402832, -0.5571545362472534, -0.2604665756225586, -1.1608586311340332, 0.8732103705406189, -0.5788953900337219, 0.7576362490653992, 0.13631048798561096, 0.7892538905143738, 1.4686754941940308, -1.4752448797225952, -0.903670072555542, -0.27365174889564514, 0.5415428876876831, 1.6871081590652466, -1.3804500102996826, -0.24088959395885468, 0.029782885685563087, 0.406000018119812, -0.6089950799942017, -0.8656510710716248, 1.0734138488769531, -0.80048668384552, 0.13206824660301208, -0.6799589991569519, 0.3548981249332428, -0.0864221528172493, -0.2976250946521759, 1.5142382383346558, 0.6592494249343872, -0.22569888830184937, 0.42649388313293457, 1.1682336330413818, 0.4732329845428467, -0.312472939491272, 0.5495062470436096, 0.904297411441803, 0.5150495171546936, -0.051794473081827164, -0.6836769580841064, -1.2744001150131226, 0.2796267867088318, 1.140838861465454, 0.004708160180598497, -0.400569885969162, -1.1771467924118042, 1.2306593656539917, 0.26081618666648865, -1.7278192043304443, -1.1775668859481812, 0.28186601400375366, 0.8263134956359863, 0.4741016924381256, -0.11539977788925171, -0.35666701197624207, 1.7074763774871826, 0.48265936970710754, -1.8480027914047241, 1.0780202150344849, 0.012383907102048397, -0.5013892650604248, -1.880123496055603, -0.4644289016723633, 0.3535890281200409, -1.6174983978271484, -0.017753640189766884, -0.2964435815811157, -0.5299630165100098, -2.3871302604675293, 0.773443877696991, -0.6134060621261597, 0.8277556300163269, 0.2099936604499817, 1.617851734161377, -0.18455512821674347, -0.3972519636154175, 1.26439368724823, 0.008118863217532635, -0.6350753307342529, -1.1040514707565308, -1.4676449298858643, 0.5515787601470947, 0.5307193994522095, 1.2492996454238892, -1.2000876665115356, 1.191358208656311, 0.46466055512428284, 1.4925283193588257, -1.0799813270568848, -0.42054083943367004, -1.8741542100906372, -2.2469706535339355, 1.523568034172058, 1.842879056930542, -1.0370640754699707, 0.8993303775787354, -0.29766571521759033, 0.640650749206543, 1.662024736404419, 1.812287449836731, -2.0017447471618652, 1.2213283777236938, -0.5719548463821411, 0.6250993609428406, 0.9669713973999023, 0.3687528073787689, -0.07720481604337692, 1.306115746498108, 0.2886674106121063, 0.5558069348335266, -0.7937471866607666, -0.0917544886469841, -0.025530725717544556, -0.48574456572532654, -0.5178101658821106, 1.100852608680725, -0.7453300356864929, 0.07517791539430618, -1.327560305595398, 0.30799704790115356, -0.6800883412361145, 0.8123747706413269, -0.13026103377342224, -0.9016105532646179, 2.1065027713775635, 0.23684453964233398, 0.22718535363674164, 0.8284094929695129, 0.9185404777526855, -0.6058712601661682, -0.11877480149269104, -1.1285130977630615, -0.7553527355194092, 0.19753457605838776, -0.23960070312023163, 0.15606853365898132, 0.08215425908565521, 0.6289229393005371, -1.208603858947754, 0.4962698519229889, 1.5544323921203613, 0.17631548643112183, -0.5285453796386719, 1.7852637767791748, 0.010029559023678303, 0.0776141956448555, -0.11528664082288742, 1.3631967306137085, 0.1002931147813797, 0.9266160726547241, -0.6386868953704834, -0.22099635004997253, -0.6320856213569641, 1.5958157777786255, 0.9822790026664734, 0.3585701584815979, -0.48703277111053467, -0.6651306748390198, -0.8806185722351074, -1.5615394115447998, -0.7597305774688721, -1.6858205795288086, -1.0504642724990845, -1.9465912580490112, 0.13192087411880493, -1.0945566892623901, -1.1841533184051514, 0.11755932122468948, -0.2179183065891266, -1.0197234153747559, -0.6870155334472656, -0.9114664793014526, -0.16821378469467163, -0.1955796480178833, -0.11892654746770859, -0.22714655101299286, 0.42500555515289307, -0.12378164380788803, 0.23716385662555695, -1.0571948289871216, -1.2154470682144165, -0.3120580017566681, 0.05835168436169624, -2.0332624912261963, 0.37978819012641907, -0.8989753127098083, 1.22779381275177, 0.38354557752609253, -1.3577518463134766, -0.8993504047393799, 1.3811955451965332, 0.15427885949611664, 0.23829670250415802, -0.3935258984565735, -1.2494475841522217, 0.10539663583040237, -0.4732532203197479, -0.7542903423309326, -1.3149408102035522, -1.263063669204712, 1.0207842588424683, -1.4233564138412476, 0.32233214378356934, -0.9039431810379028, -0.3775078058242798, -0.8021509051322937, -0.08299537003040314, -1.0466073751449585, -0.017247669398784637, -1.7998017072677612, 0.47507452964782715, 0.784919023513794, 1.1497695446014404, 0.31097787618637085, -0.2563343346118927, -0.7933169603347778, 1.4568942785263062, 0.4035235345363617, -1.2632728815078735, 2.5122838020324707, -1.0049742460250854, -1.6396209001541138, 0.8482876420021057, -1.1120903491973877, 1.4646354913711548, 0.0383952371776104, -0.07622580230236053, -0.38974136114120483, 0.03275437653064728, -0.27719205617904663, -0.3290410339832306, -0.6752545237541199, -1.938336968421936, 1.9588520526885986, 0.3528386354446411, -0.41488635540008545, -0.10202201455831528, 0.8516354560852051, 0.4780963957309723, -0.2599497437477112, -0.7527973055839539, 2.9821860790252686, -0.11703061312437057, -0.988599419593811, -0.7428978085517883, -0.31396597623825073, -1.343515157699585, 0.6661078929901123, -0.6584908366203308, -0.23797079920768738, 0.5062987208366394, -0.6153355836868286, 0.1445198506116867, -0.2145688235759735, 1.701983094215393, 1.5231722593307495, -0.16887526214122772, -1.0821584463119507, 0.3627568781375885, -0.09896302968263626, 0.5703520178794861, -0.48774436116218567, -1.053445816040039, -0.5975735187530518, 1.6400412321090698, -0.7362219095230103, 0.6311963200569153, -1.8732012510299683, -0.059514369815588, -0.9210812449455261, -0.39298224449157715, -0.8593926429748535, -1.6723837852478027, 0.12489809095859528, -1.0140049457550049, 0.479557067155838, 0.2891053557395935, 1.0116416215896606, -1.2691131830215454, 1.3657437562942505, -1.0604324340820312, -0.09408529847860336, 0.1638437807559967, 0.948935866355896, -1.4533649682998657, -1.2386430501937866, -0.2816075086593628, -1.6783517599105835, -0.07816483080387115, -0.3176516890525818, -0.8985844850540161, -0.7680283188819885, -0.8632646799087524, -1.0033540725708008, 1.4698525667190552, -0.6886643171310425, 0.5668947100639343, 0.18885856866836548, -0.2005576640367508, -0.2885521650314331, -1.2248268127441406, -1.3703587055206299, -0.3041157126426697, 0.8020142912864685, -0.6699432134628296, -0.4862913191318512, 0.6513763666152954, 1.4435088634490967, 0.8631761074066162, -0.8948178887367249, 0.10401708632707596, 1.2565696239471436, -0.5664752125740051, 0.7382693886756897, -1.106081247329712, 0.7546799182891846, 1.106489896774292, 0.1146206259727478, -0.4343865215778351, -0.4333057403564453, 0.18956005573272705, -0.6917535662651062, -0.9496145248413086, -0.036509331315755844, 0.8204907774925232, 1.8282828330993652, 0.4203498959541321, -1.9515372514724731, -0.8113678693771362, 1.9570386409759521, 0.06269751489162445, 0.33659255504608154, -0.37934279441833496, 1.9083629846572876, -0.7437235116958618, -0.1969848871231079, 0.8212971687316895, -0.32535362243652344, 0.3974027931690216, 0.7305675745010376, -0.23215000331401825, -0.0006430019275285304, 0.9879686832427979, 2.338881254196167, 0.21927382051944733, -0.31266599893569946], [0.31056880950927734, 1.8734651803970337, 0.2929934561252594, 0.6638692617416382, -0.7704445719718933, 0.2350730448961258, -1.5255235433578491, 0.15916389226913452, 1.0910801887512207, -0.09778087586164474, 0.11365339159965515, 0.10299701243638992, -1.2880568504333496, 0.8082157373428345, 0.19154886901378632, -0.03014177829027176, -0.2566509544849396, 0.5502626895904541, 2.5379252433776855, -1.0551098585128784, 1.8376230001449585, -0.18796516954898834, 0.7003757357597351, -0.2443675547838211, 0.19318655133247375, 0.35302895307540894, -1.5113422870635986, -1.3559517860412598, 1.4765921831130981, 1.1196997165679932, 0.18472422659397125, 0.34725260734558105, -0.17581722140312195, -0.24879522621631622, -1.1159143447875977, -0.13289949297904968, -0.8179287314414978, -0.04005030542612076, -0.13585086166858673, -0.01645766757428646, -0.3113221228122711, 1.7058842182159424, 0.5734599232673645, 1.3890293836593628, 0.21671417355537415, -0.7104634642601013, -1.1759346723556519, 0.39944547414779663, -0.2165389209985733, 0.879887580871582, -0.4679234027862549, -1.1583739519119263, -0.3209995627403259, 0.7993475794792175, 1.2567366361618042, -0.0033256919123232365, -0.1609402447938919, -0.20352357625961304, 0.8397471904754639, -0.326616108417511, 0.8855919241905212, -1.1805803775787354, -0.8129487037658691, -3.214332342147827, -0.36830976605415344, 0.8434870839118958, -0.5677400231361389, -1.3504823446273804, 0.015500909648835659, -0.6358139514923096, -0.9991664290428162, -0.398565411567688, 0.049982551485300064, 1.2814091444015503, 0.39476263523101807, -0.3011302351951599, -1.3190377950668335, -0.8625047206878662, 0.02792813815176487, 0.8785064220428467, 1.8399466276168823, 0.8107975721359253, 3.5658228397369385, -0.8115007877349854, 1.9007779359817505, 0.45362234115600586, -0.7232025861740112, -0.21372966468334198, 1.1801674365997314, -0.2156105786561966, -0.0015678172931075096, 0.5720458030700684, -0.16683021187782288, -1.1130577325820923, -0.7075290679931641, -2.191455602645874, -0.704927384853363, -1.5697439908981323, 1.057388186454773, -1.3966419696807861, -0.0888969823718071, 0.5226221084594727, 0.18319639563560486, -0.5241901278495789, 1.1027635335922241, -0.07791157066822052, 0.04561562091112137, -0.8097927570343018, -1.0525308847427368, 1.2785700559616089, 0.17198126018047333, 0.15040841698646545, 0.5514439344406128, -0.2384892702102661, -0.6277571320533752, 1.9087927341461182, -0.7554525136947632, -0.0065514142625033855, 0.5317257046699524, 0.5041139125823975, -1.7421501874923706, 2.5409867763519287, 0.19704197347164154, -0.6728519201278687, -0.15206462144851685, -1.1594678163528442, -1.031929612159729, -0.22935707867145538, 0.006636621430516243, 0.13432784378528595, -0.08462744951248169, -0.4522306025028229, 0.0317339226603508, -0.3833880126476288, 0.24820677936077118, 0.23615522682666779, -0.5664400458335876, -1.8591328859329224, -1.407266616821289, 0.33634084463119507, -0.26354941725730896, -1.2368762493133545, 0.7570116519927979, 0.18071378767490387, 1.025334358215332, 2.129556655883789, 0.2905780076980591, 1.209707260131836, -0.22084856033325195, -0.4806227385997772, 0.3349730670452118, 0.6312333345413208, -0.314590185880661, -0.3215222656726837, -0.47962021827697754, -1.209053874015808, 0.5954908728599548, 0.9287446737289429, -0.7303497195243835, 1.6716384887695312, -0.8046379685401917, 0.375403493642807, -1.7174625396728516, 0.2905746400356293, -0.6812552809715271, -0.13065700232982635, -0.30081233382225037, 0.46573829650878906, -0.3611406981945038, -0.4011453092098236, 0.7032492756843567, -0.546235203742981, 0.8907152414321899, 2.056535482406616, 2.195293426513672, 2.0116348266601562, -0.15108586847782135, -0.019337017089128494, 0.48940300941467285, -0.03068072907626629, -1.26558518409729, 0.8314725160598755, 0.9467877745628357, -0.5777190327644348, 0.43217551708221436, 1.595962405204773, -0.15295274555683136, -0.8232548236846924, 1.432671308517456, -0.22999253869056702, -1.0159876346588135, -1.487939715385437, 0.4003989100456238, -0.03973681107163429, 1.8481650352478027, -1.0072650909423828, -0.403812974691391, 1.7629187107086182, -1.3606921434402466, -0.13174816966056824, -0.23212294280529022, -0.0390031635761261, -0.6901102066040039, -1.338127613067627, 0.939795732498169, 0.40990376472473145, -0.08915676921606064, 0.4669344127178192, 1.508868932723999, 0.14969396591186523, -1.3237699270248413, 1.2133432626724243, 0.18036876618862152, -0.5956003069877625, -0.8683830499649048, -0.31113719940185547, 1.472442865371704, -0.03971245512366295, 0.09819406270980835, 1.5788085460662842, -0.30953726172447205, -1.832821011543274, 1.5319803953170776, 0.36141085624694824, 0.03535495698451996, -1.0974005460739136, -1.2891212701797485, -0.5445731282234192, -0.4622826874256134, -0.564758837223053, -0.09495532512664795, 0.5927287340164185, -0.7752728462219238, 0.8754895925521851, 0.6742486953735352, -0.8679684996604919, 0.012117516249418259, 0.586548924446106, -1.9324482679367065, 0.03440667316317558, 1.1827569007873535, 0.09829537570476532, -0.3084791302680969, -0.9938599467277527, 0.7890374064445496, 0.4958391487598419, -0.789584755897522, -0.20676182210445404, 0.27689623832702637, -0.7839636206626892, -0.45835354924201965, -0.06542986631393433, -0.5102845430374146, 1.0268045663833618, 1.6348048448562622, -0.4814298152923584, 0.626552164554596, -0.9350831508636475, 1.687510371208191, 0.52569580078125, -0.42324623465538025, -0.9667952060699463, -0.7375686764717102, -0.48625755310058594, -0.4710210859775543, -0.3965078294277191, -0.32621586322784424, -1.156747579574585, 1.2457022666931152, -0.01666027121245861, -0.4498335123062134, 1.6206706762313843, 0.8965221047401428, -0.7834270000457764, -0.16846127808094025, 0.848163366317749, -1.2623878717422485, 1.5790343284606934, 0.5724824666976929, -1.2919493913650513, 0.6762334704399109, -0.8252608776092529, 0.9366596937179565, 0.27829164266586304, 0.747548520565033, -0.5505192875862122, -0.4164237380027771, 1.6081087589263916, 1.8920061588287354, -0.983383297920227, -0.44187286496162415, -0.7386153340339661, 0.38077032566070557, -0.13104678690433502, -0.340329110622406, -0.5459355115890503, -0.1046789214015007, -1.4020702838897705, 1.9859287738800049, 0.28608331084251404, -1.1313927173614502, -0.6510618925094604, 0.1613578200340271, -0.3617233633995056, 0.263871431350708, -0.5746550559997559, 0.8770140409469604, -1.3833739757537842, 0.6543459892272949, 0.6138196587562561, -0.34860333800315857, 0.02067747339606285, -0.9686034321784973, 0.7994648218154907, 1.370840311050415, -0.09128177911043167, 0.19318991899490356, 0.4625742435455322, 0.7436139583587646, -0.09850458055734634, 0.3472473919391632, 0.8041011095046997, -0.5306078195571899, -0.8443843722343445, 0.6409987807273865, 1.0633783340454102, -0.4794890582561493, 0.48054084181785583, -1.4765801429748535, -1.4102734327316284, 0.17794041335582733, -0.017466170713305473, 1.4071708917617798, 1.8015384674072266, -0.4955076277256012, -0.0702953040599823, -0.9654965400695801, -1.152235984802246, -0.7984811067581177, 0.38311028480529785, 0.4090019166469574, 0.7813672423362732, -0.45680588483810425, -0.4189440608024597, 1.2698074579238892, -0.3130056858062744, -0.5602965950965881, 0.026215048506855965, 0.931090235710144, 0.09835593402385712, -0.46344661712646484, 0.07794354856014252, -0.7058846354484558, -0.8829912543296814, 0.7441229820251465, -0.3254815340042114, -1.3332468271255493, -1.7677463293075562, -1.0997332334518433, 0.36773189902305603, -0.055446185171604156, 0.7541823387145996, 0.854869544506073, -0.44780805706977844, -0.6611449122428894, 0.140013188123703, -0.23156790435314178, 0.44182220101356506, -1.0361456871032715, 0.4526174068450928, 0.7419835329055786, 1.2377336025238037, 0.062428466975688934, -0.8736676573753357, 3.7518906593322754, 0.05609500780701637, 0.2341155856847763, -0.2850739657878876, -0.9104192852973938, -0.23607364296913147, -0.5283744931221008, 0.8986393809318542, 0.39258521795272827, -1.045601725578308, 2.06265926361084, 0.5218179821968079, -0.4608043432235718, -0.2449672818183899, -1.7442262172698975, 0.7003986239433289, -0.24829553067684174, 0.34265604615211487, -0.14542913436889648, 1.2719452381134033, 0.7265322804450989, -1.1644551753997803, 1.238050103187561, -0.66763836145401, -0.40883320569992065, 0.26133963465690613, -0.25144824385643005, -0.6348147392272949, -0.28526875376701355, 1.469462275505066, -0.7315013408660889, 1.2092686891555786, -1.4445085525512695, 0.06278786808252335, 0.6725905537605286, -0.13355609774589539, -0.9920287132263184, 1.3882657289505005, 0.7192198038101196, 0.5953326225280762, -0.19053080677986145, 0.3082262873649597, -0.135869100689888, -0.024526504799723625, -0.6790074706077576, -0.3025776147842407, -2.16959285736084, -2.367206573486328, -0.8827794790267944, -0.7591278553009033, -0.3325515389442444, -2.30778169631958, 0.9930506944656372, -0.19002003967761993, 0.2862786650657654, 0.081448994576931, 0.21904142200946808, 0.23231221735477448, 1.6002410650253296, -1.411628007888794, 0.7521880269050598, -0.9062367081642151, 0.46851736307144165, -0.6379570364952087, 0.4227854013442993, 0.16752378642559052, 0.589867889881134, 1.3066457509994507, 0.65944504737854, -0.24458913505077362, -0.4943954646587372, 1.518625259399414, -0.9242013096809387, 0.5144351720809937, -0.07137420773506165, 0.5305166840553284, -0.1582086980342865, 0.582257866859436, -0.454409122467041, 0.27709662914276123, -1.2045389413833618, 0.41168832778930664, 0.23390065133571625, -0.9276022911071777, -0.441638320684433, 0.03333621099591255, -0.5825064778327942, 1.263288974761963, 0.3061646819114685, 0.24092112481594086, -1.1927918195724487, 0.501600980758667, 0.36066344380378723, -0.08537296205759048, 1.0243889093399048, 0.6408947706222534, 0.8631001710891724, 0.5921609401702881, -2.776055335998535, -0.10048776119947433, 0.26423460245132446, -0.21613629162311554, -1.454520344734192, 0.9911103844642639, -0.27187567949295044, -0.026181403547525406, 0.2924453318119049, -1.323094367980957, 0.2851790189743042, -0.3270896375179291, 0.26002755761146545, -0.602354884147644, 0.9741918444633484, -1.3345324993133545, 0.29842182993888855, 0.40479549765586853, 0.972637951374054, -1.0147377252578735, -0.1305699646472931, -0.3297707438468933, -0.4741179347038269, 0.66057950258255, -1.7430181503295898, -0.034819986671209335, -0.019803641363978386, -0.6625910401344299, 0.5320062041282654, -0.5999438166618347, 0.5295743346214294, 0.8321360349655151, -0.4482375681400299, 0.08505245298147202, -0.6378176212310791, -0.06482444703578949, 0.8738841414451599, -0.8073725700378418, -0.1551392674446106, 0.93265300989151, 1.4344826936721802, -1.0073556900024414, 0.5144530534744263, 0.3453153669834137, 0.571675181388855, 0.3431626856327057, 0.3952609896659851, -0.16284491121768951, 0.7163277268409729, -0.2784866392612457, -0.23701626062393188, 1.226670265197754, -0.45981523394584656, -0.4269706606864929, 0.5193250775337219, 0.7325156927108765, -1.0316493511199951, -1.2744250297546387, 1.0765033960342407, -0.21867015957832336, 0.43967029452323914, 2.4270689487457275, 1.1737573146820068, 0.8899245262145996, -0.21868224442005157, 1.962320327758789, 0.4500633776187897, -0.09131141752004623, -1.181938886642456, 0.33220136165618896, -1.8747191429138184, 0.29943007230758667, 2.3657517433166504, 0.8785183429718018, -1.340423345565796, 0.18949246406555176, -0.14002367854118347, 1.2129758596420288, 0.760891854763031, 1.024735450744629, 0.8627044558525085, 0.8341209292411804, 0.18879494071006775, -0.9834189414978027, 1.268730878829956, 0.560474693775177, 0.6405999660491943, 0.3949565291404724, 0.1206577867269516, 0.44587045907974243, 0.04728669673204422, -0.6713254451751709, 0.3864874541759491, -0.41865473985671997, -0.5354558229446411, -0.18258802592754364, -0.4061906933784485, -0.764708399772644, -0.03338339552283287, -0.9943784475326538, -1.4226995706558228, -0.46996626257896423, -2.835329532623291, -1.2577097415924072, 0.20223741233348846, -0.4640386700630188, 0.6971454620361328, -1.3079193830490112, 1.3710285425186157, -0.34790360927581787, -0.63868647813797, -0.369467556476593, 2.2305707931518555, -0.08786336332559586, -1.874998927116394, 0.5679641962051392, -0.15318071842193604, 1.7208747863769531, 0.006418358534574509, 0.7397117018699646, 0.05160311236977577, 0.17019601166248322, -0.06139659881591797, -0.8017899394035339, -1.2269428968429565, 1.2120184898376465, 0.01778874546289444, -0.049383677542209625, -0.33318114280700684, 0.43649357557296753, 0.7100510597229004, 0.5984265208244324, -0.3192138671875, 0.1335821896791458, -0.610499918460846, -0.47884631156921387, 1.2692170143127441, 0.2863783836364746, 0.21225403249263763, 0.1215188279747963, -0.9894497990608215, 1.0928312540054321, -1.2864428758621216, -0.3558970093727112, -0.06753924489021301, 2.0660812854766846, -0.9758455157279968, -1.090928554534912, -1.5873135328292847, 0.41365620493888855, -0.22080519795417786, 0.0861545205116272, -1.3350636959075928, -0.12388493865728378, -0.5452525019645691, -1.2245395183563232, 1.08241605758667, -0.20342718064785004, 0.954397976398468, -0.988760232925415, -0.15379491448402405, 0.3214327096939087, 0.026328828185796738, -1.2114627361297607, 0.6857750415802002, 1.751074194908142, 1.524479627609253, -1.3738690614700317, 0.79603511095047, 0.028353774920105934, -0.09758453071117401, -0.18733876943588257, 0.5679794549942017, 0.20202068984508514, 0.11193432658910751, -1.5200351476669312, 0.7637171745300293, -1.322495937347412, 1.4849117994308472, -0.3619551658630371, -0.8222376704216003, 0.5712665915489197, 0.2744649052619934, -2.3835833072662354, -1.636047601699829, 0.46905890107154846, -1.1526215076446533, 0.2922883629798889, 0.05969052016735077, -0.6155455112457275, -0.6090068817138672, 0.3682284951210022, -1.1739400625228882, -0.6054903268814087, -1.3624755144119263, 0.21169476211071014, -0.15744082629680634, 0.3834816515445709, -1.7761027812957764, -0.6187182664871216, -1.0295124053955078, -0.35423314571380615, 0.9018791913986206, -0.3425537943840027, -0.6996655464172363, -0.36069178581237793, -0.6241675019264221, 0.6074554324150085, 0.7399449944496155, 0.48530343174934387, -0.07184800505638123, -1.0018162727355957, 1.751681923866272, -1.390339732170105, -0.3442544639110565, 1.2890013456344604, -0.3714964687824249, 1.795904278755188, 1.7294495105743408, -0.5769425630569458, 0.09998057782649994, 0.5025177597999573, 0.8101003170013428, 3.1005895137786865, 0.710942268371582, 1.2683782577514648, -0.06018868461251259, 0.1508888453245163, 0.005082071293145418, 0.27372321486473083, -2.054617404937744, -1.7586137056350708, -0.26265835762023926, 0.9689270257949829, 0.06606355309486389, -2.8676412105560303, -0.5203542113304138, -1.4589439630508423, 1.1679081916809082, -0.29607948660850525, -0.6052119135856628, 0.14706265926361084, -0.49141445755958557, 1.3953077793121338, -1.0613608360290527, -0.3680751323699951, -0.2867105305194855, 0.3678470849990845, -1.588308572769165, -0.01426242757588625, 0.5606955289840698, -0.09240880608558655, -0.2274361550807953, 0.6933834552764893, -0.6651429533958435, 0.9971528649330139, -0.19827628135681152, -0.650593101978302, 0.5222397446632385, 1.8714343309402466, -0.023835349828004837, 0.4298502802848816, 0.0893278494477272, 0.2201816439628601, 1.7983438968658447, 0.6643873453140259, -0.8972586989402771, 0.7198182940483093, 1.3643862009048462, -1.2029495239257812, 0.1782408505678177, -0.05393921211361885, 0.650883138179779, -0.03726271539926529, -0.3811708092689514, 0.61492919921875, -0.11544611304998398, 0.2292688935995102, -2.1546363830566406, 0.10462332516908646, 0.5658562183380127, 0.6756010055541992, 0.46917295455932617, -2.1115658283233643, -2.261446952819824, 0.6364629864692688, -2.68559193611145, 0.9606910943984985, 1.1071940660476685, 0.06820215284824371, 0.1967463493347168, -0.31490248441696167, -1.9947885274887085, -1.313546061515808, -1.0194178819656372, -0.22934263944625854, -0.08688299357891083, 0.2799154222011566, 1.2011173963546753, -1.1987278461456299, 0.468925803899765, 0.9518740773200989, -2.136035442352295, 0.7317922711372375, 1.1747628450393677, -0.16593818366527557, -0.45313161611557007, 1.1441370248794556, 0.44297003746032715, -0.8050273060798645, 0.23228204250335693, -0.6416992545127869, -2.4121813774108887, -0.6755538582801819, 1.2919375896453857, -0.22539083659648895, 0.1098741739988327, 0.0790335014462471, 1.5692384243011475, -1.1908674240112305, 0.09354674071073532, -0.6558991074562073, -0.11152949184179306, -1.1119338274002075, 0.36167824268341064, -1.8460344076156616, -0.7154639363288879, 1.2393587827682495, 0.2214641273021698, 1.3404909372329712, -0.019624728709459305, -0.5456715226173401, -0.9400162100791931, -1.5294705629348755, -0.3171135485172272, -0.47356322407722473, 0.6226059794425964, -0.16822253167629242, 2.19916033744812, -0.2826994061470032, 0.7269700765609741, 1.4165047407150269, 0.093251533806324, -0.2683587074279785, 0.8731931447982788, 0.8662168383598328, 2.7409799098968506, 0.38089171051979065, -0.1710512936115265, -1.27780282497406, 1.1596134901046753, 1.8870882987976074, 2.1918601989746094, 0.4923192262649536, -0.30710288882255554, 0.1483951359987259, -1.2194292545318604, -2.0653278827667236, -0.06877996027469635, -0.7925494909286499, -1.0334738492965698, 0.08441480994224548, -0.8599803447723389, -0.34269917011260986, -1.2676877975463867, -2.355536699295044, -0.5986800193786621, 0.1617555171251297, 0.23285804688930511, 0.9598386287689209, -0.5331792831420898, 0.9949216246604919, -0.10406843572854996, -1.9603463411331177, -1.8534632921218872, 0.9239531755447388, -0.5275830626487732, -0.6449987292289734, -0.6616945266723633, -1.7169357538223267, -0.9154402017593384, -2.031712293624878, 0.09029525518417358, 0.5775153040885925, 0.24700170755386353, -0.3267635107040405, 1.7634221315383911, 0.030811088159680367, -0.02669789455831051, 2.176194190979004, -1.4564003944396973, -1.582666039466858, 0.6489811539649963, -0.32171452045440674, 0.7790711522102356, -0.8785306811332703, -0.6663249135017395, -1.4601917266845703, 1.024429440498352, 0.8521929383277893, 1.1588375568389893, 0.6256765723228455, 0.48414403200149536, -0.40995192527770996, 0.592673122882843, 0.16962091624736786, -0.5058767795562744, 0.9147947430610657, 0.7957826256752014, 1.1652659177780151, 1.84176766872406, -2.415766716003418, 0.492859423160553, 0.3663714528083801, -0.41980376839637756, 0.4748125970363617, 0.8343517184257507, 1.751414179801941, 1.1063814163208008, 0.6284347176551819, 0.6667455434799194, 0.6958515048027039, 0.9681733846664429, -0.060349367558956146, -1.0735535621643066, -0.5582727193832397, -0.0819433182477951, 0.3898538053035736, 0.26077577471733093, 0.5343348383903503, -0.6554126143455505, -1.3976253271102905, 0.02640964835882187, -1.4347941875457764, -0.11085719615221024, -1.6294338703155518, -0.9640387892723083, 0.6632025241851807, -1.1643257141113281, -0.4758830964565277, -0.6808935403823853, -0.8062593936920166, -0.7769755125045776, -0.27146705985069275, -1.1127883195877075, 0.5337561368942261, 1.099076271057129, 0.05757663771510124, -0.3910159468650818, 0.8227620720863342, 1.4330649375915527, 0.5332188010215759, -1.8764830827713013, -0.814348578453064, -0.26747751235961914, 0.43726521730422974, 0.8502592444419861, -0.013023034669458866, -1.158504843711853, -1.4583797454833984, 0.07100215554237366, -2.3571014404296875, 1.1803151369094849, 0.8022012114524841, -1.4786152839660645, -0.6713275909423828, -1.3567888736724854, 1.0762038230895996, 0.7996429204940796, 0.3594001233577728, 1.153476357460022, 0.8374719023704529, -1.7114534378051758, -0.5172874927520752, 1.1728700399398804, -2.438344717025757, 2.2935636043548584, 0.25313225388526917, 0.24090087413787842, -0.014587601646780968, -0.3639810383319855, -0.56609708070755, -1.5815147161483765, -0.23667319118976593, 0.5274189710617065, 0.18184535205364227, 0.14111004769802094, 0.41351309418678284, 0.9768287539482117, -0.2799903452396393, 1.9485739469528198, 0.15252363681793213, -0.08574331551790237, 1.040532112121582, 0.05690481513738632, 1.0202610492706299, 0.20360036194324493, -0.25089746713638306, 0.5880296230316162, 0.9656932950019836, -0.428609699010849, 0.5721999406814575, 0.4265667498111725, 1.3325731754302979, -1.2618480920791626, 0.4438701570034027, 0.5144286751747131, -1.098677396774292, 0.43576177954673767, 0.6199924349784851, 0.22525568306446075, 0.6171507835388184, -0.1692311316728592, 1.385305643081665, 0.24068036675453186, -0.8394088745117188, -1.313741683959961, -0.21434073150157928, 0.23749613761901855, -0.8494540452957153, -0.5293473601341248, -0.5141573548316956, 1.12931489944458, -0.44259265065193176, 1.1422007083892822, 1.5445544719696045, -0.3249781131744385, -1.1345239877700806, 1.1964658498764038, -0.04782666638493538, -1.9125208854675293, 0.3420358896255493, -0.1719224900007248, 0.33780670166015625, -1.7254999876022339, 0.8237072229385376, 0.03770294040441513, -0.8055776953697205, 0.0324307382106781, 1.993844747543335, -0.8601342439651489, -0.8032979965209961, 1.7448936700820923, -0.890632688999176, -0.7199530005455017, -0.5409488677978516, -0.049950551241636276, -0.37287402153015137, 0.41929495334625244, -2.4674670696258545, -0.17830578982830048, 0.21598437428474426], [1.2383486032485962, -1.2767945528030396, -1.1045091152191162, -0.45871514081954956, -0.5499494671821594, -0.4336629807949066, 1.066641092300415, -0.10091570764780045, 1.072241187095642, 0.5245112776756287, 0.09850776940584183, -0.8452015519142151, -1.5519869327545166, 0.06491517275571823, -0.14061859250068665, -1.4527772665023804, 0.7169678211212158, -0.9140586256980896, 1.3032307624816895, -0.23139943182468414, -1.015769124031067, 0.706156313419342, 1.673891544342041, 0.35133156180381775, -2.0681395530700684, -1.4260215759277344, 0.24743013083934784, 0.7574899196624756, 1.2906434535980225, 0.3707309663295746, 0.15722250938415527, 0.18684901297092438, 1.0417481660842896, 0.5678949952125549, 0.859563410282135, -0.36630985140800476, 0.08377296477556229, -0.44341450929641724, 0.627297043800354, -1.0664033889770508, -0.8428078889846802, 1.63145911693573, -0.6420366168022156, -0.851813554763794, -0.023376578465104103, 1.6137906312942505, 1.513370156288147, -0.6819000244140625, -0.5615613460540771, 0.47732025384902954, 1.4515870809555054, -0.6229056715965271, 0.04547156020998955, -1.2051026821136475, -0.9987147450447083, 0.050563912838697433, -0.018405979499220848, -2.0969557762145996, 1.3077917098999023, 0.8780360221862793, -1.5560333728790283, 1.419127345085144, -1.3683850765228271, -0.26004984974861145, -1.3017840385437012, -1.082062840461731, -0.1497291475534439, -0.30049899220466614, 0.3955364227294922, -0.21866878867149353, -0.20943741500377655, -0.7461841106414795, 1.5837013721466064, 0.49299994111061096, -0.32012301683425903, 0.4075869917869568, 0.4220375716686249, -0.9454973936080933, -1.3914334774017334, -1.5253150463104248, 0.5847595930099487, -0.8308711051940918, 0.7194546461105347, 0.9001577496528625, -0.5933048129081726, 0.8285662531852722, -0.28470897674560547, -1.0914608240127563, -0.5548319816589355, 0.6708812713623047, -0.7608558535575867, 1.034206748008728, 0.05824850872159004, -1.2405518293380737, -2.2113521099090576, -0.37777963280677795, 0.3295287787914276, -0.5750133395195007, 1.3670085668563843, 1.858072280883789, -2.234891653060913, -0.421749085187912, 0.7725203037261963, 0.6489034295082092, 0.33006608486175537, -1.0660858154296875, -0.423450231552124, 0.5405216217041016, 2.6406967639923096, -0.8411044478416443, -1.33091402053833, 2.312598466873169, 1.13973069190979, 0.655535876750946, -0.7890300154685974, 0.7348564863204956, -1.4342155456542969, 1.299787998199463, 0.713882565498352, -0.18815088272094727, 0.13043048977851868, 0.9178106784820557, 0.714664876461029, 0.5624257922172546, -0.6323164105415344, 1.0506819486618042, -0.2160978764295578, 0.06506370007991791, 2.2939703464508057, -1.612076759338379, -1.020071029663086, 0.519000768661499, 1.50946843624115, -0.19743815064430237, -1.3895448446273804, 0.6904621720314026, 0.33340951800346375, -0.18550711870193481, 0.8678522706031799, 0.9090025424957275, 1.2765941619873047, -0.4585164487361908, 0.890362024307251, -0.9623939394950867, -1.001827597618103, 0.14045612514019012, -1.0088306665420532, 0.26105383038520813, 0.9971429109573364, -1.3235392570495605, 0.07346878200769424, 0.42031538486480713, -0.28888872265815735, -0.6861189603805542, 0.07587698847055435, -0.8865779042243958, 0.5143706202507019, 1.4917607307434082, -0.8360159397125244, -0.7499241232872009, 0.9477588534355164, 1.3015977144241333, -0.44866272807121277, 0.7529584169387817, 1.9119149446487427, -0.9260284304618835, -0.9298049807548523, -0.46168726682662964, 0.08606164902448654, 1.1054600477218628, 1.2350304126739502, -0.055974941700696945, -0.9792851805686951, 0.7978838086128235, -0.8825966119766235, -1.028825283050537, -0.5472671389579773, 0.24118715524673462, 0.5226870775222778, -0.7203776240348816, -0.635220468044281, -2.0061135292053223, 0.5043717622756958, -1.24905264377594, -0.45984089374542236, 0.8971337676048279, -0.6527084112167358, -0.8478963375091553, 0.1340467482805252, 0.7760708332061768, 0.4806966781616211, 0.7970460057258606, 2.9444165229797363, 1.086328387260437, -0.7325761318206787, -0.6235509514808655, 1.1296602487564087, -0.11466587334871292, -0.24614128470420837, -0.20333872735500336, -0.9328513145446777, 0.5790930390357971, 0.19670705497264862, -0.2985820472240448, -0.47096848487854004, 1.2762596607208252, -0.4353298544883728, 1.784530758857727, 0.9644762277603149, 0.011414211243391037, 0.20936165750026703, -1.5579414367675781, -2.112514019012451, 1.68307626247406, 1.0153617858886719, -1.571416974067688, 0.10618584603071213, -0.3180258870124817, -0.7937362194061279, -1.1226961612701416, -0.4784191846847534, 0.7186018824577332, -0.2177022397518158, -1.8597332239151, -0.992010772228241, -2.6223199367523193, -0.2667294442653656, 0.0674397200345993, 1.7577309608459473, 0.7963372468948364, -1.0505400896072388, -0.022410791367292404, -1.5043644905090332, -0.4714259207248688, 0.26663413643836975, 1.2216767072677612, 1.6758302450180054, -0.42058563232421875, 2.985564947128296, -0.3472300171852112, -1.165520429611206, 1.2902427911758423, 0.7711560130119324, 0.3210728168487549, -0.3844248056411743, -1.4655283689498901, -0.8408591747283936, -0.9510414004325867, 1.2375000715255737, -0.7037074565887451, 0.16256383061408997, 0.9709672927856445, -2.155925750732422, -0.02138187736272812, 0.13683843612670898, 1.1852216720581055, 0.6730325222015381, 0.006884480826556683, -1.3185945749282837, -1.016187071800232, -0.5797660946846008, 1.4324219226837158, -1.3638935089111328, 0.1609325408935547, -1.5738576650619507, 0.18827693164348602, -1.6074687242507935, 0.5625535845756531, -0.9343562126159668, 1.1211771965026855, -0.8151825070381165, -1.1338815689086914, 0.9657616019248962, 0.3571222424507141, 0.2591478228569031, 1.4256962537765503, -0.7346627712249756, -0.12283892929553986, 1.199104905128479, 0.19785355031490326, -0.7773693203926086, -0.15064965188503265, 0.27313166856765747, -0.22546686232089996, -2.1195526123046875, 0.14709314703941345, 2.3661792278289795, 0.5341259241104126, -0.13057591021060944, -0.3557504713535309, -0.1041855588555336, 0.4933391511440277, -2.1672024726867676, 0.11185776442289352, -1.1810392141342163, 0.9884710311889648, -0.49164262413978577, 1.306374430656433, -0.10603369772434235, 0.8341636657714844, 0.5332863330841064, 2.3920304775238037, -0.9959738850593567, -0.5238246917724609, 1.1571341753005981, -0.7417100667953491, -1.0074387788772583, 1.8011406660079956, -0.3438262939453125, -0.25734105706214905, 2.1594340801239014, -0.9457074403762817, 1.2822797298431396, -0.05547178164124489, 1.9631356000900269, 0.9439982175827026, -0.35472357273101807, 0.6566919088363647, -0.5820726156234741, 1.7295037508010864, 0.2891109585762024, -0.9950193166732788, 0.36801081895828247, -0.9370308518409729, -0.17268304526805878, -0.18444062769412994, -1.6491035223007202, 0.3941042721271515, -1.3227334022521973, -2.7047932147979736, 0.12243097275495529, 1.2335200309753418, -1.7206064462661743, -0.8724706768989563, 0.6379573345184326, -0.19928032159805298, -0.19770358502864838, -0.9531534314155579, -3.478846788406372, 0.22559741139411926, -0.15526290237903595, 0.1205357164144516, 0.01967385783791542, 0.35479095578193665, 0.09627343714237213, 0.9814589619636536, 0.41779595613479614, -0.7207465767860413, 2.500494956970215, -0.4041880965232849, -1.177703619003296, 0.4021000862121582, 0.586756706237793, -1.0290718078613281, 0.4969669580459595, -1.315213918685913, 0.7365719676017761, 0.914192259311676, -1.1037437915802002, 1.4958488941192627, 2.5308239459991455, 0.77667236328125, -0.34283989667892456, 0.7098861336708069, 1.3382500410079956, -0.6577414274215698, -1.008718490600586, 0.8920595049858093, 0.9159544706344604, -0.861764669418335, 0.808688759803772, -0.01368188951164484, -1.1065192222595215, 0.3345346748828888, 0.074245385825634, 2.3511271476745605, 0.6887533068656921, -0.9963120222091675, -1.8611412048339844, -0.01968102902173996, 1.5986922979354858, -0.6672360301017761, -0.41688692569732666, -0.3812516927719116, 0.9318789839744568, -0.3197735548019409, -0.6368083357810974, -0.18408726155757904, 0.6802928447723389, 1.2760306596755981, -0.36294785141944885, 0.7865840792655945, 0.12905332446098328, -0.19246569275856018, 0.6469250321388245, 1.921035885810852, 0.7761016488075256, 0.08321215957403183, -1.1442358493804932, 2.0365278720855713, -0.047611434012651443, 0.7845174074172974, 0.6900883913040161, -1.7606450319290161, -1.8765403032302856, -0.1145472526550293, -0.429502934217453, -0.6318047642707825, -0.478444367647171, -0.9131557941436768, -0.4660182595252991, -1.5497075319290161, 0.6771478056907654, 0.30605340003967285, -0.41694748401641846, 0.05011586472392082, -1.1035256385803223, 2.869316577911377, -0.05110693350434303, 1.9889376163482666, -1.7764389514923096, -0.4971875548362732, -0.3367297053337097, 1.157752275466919, 0.955190122127533, 1.5591202974319458, -0.053684890270233154, -0.43949633836746216, 1.8660873174667358, 2.4441888332366943, -1.4728293418884277, 1.1556657552719116, 0.7635213136672974, 0.28947341442108154, -1.633481502532959, -0.25052228569984436, 1.0895572900772095, -1.6286897659301758, 0.7742799520492554, -1.2996281385421753, 0.8061102628707886, 0.5707212686538696, -0.34297361969947815, -0.38708674907684326, 0.3494327962398529, -1.375237226486206, 2.5535600185394287, 0.02829100377857685, -0.4722973704338074, -1.4113551378250122, -0.5839688777923584, 0.5932698845863342, -0.8189558982849121, 0.5005776286125183, -2.0053629875183105, -0.2291383445262909, 0.3215380907058716, 0.7989878058433533, -0.5636140704154968, -0.3893825113773346, -0.04654495418071747, -0.20716749131679535, -0.5809834003448486, -0.7742224335670471, 0.7172876596450806, -0.09492509067058563, 1.3023229837417603, 0.08780409395694733, -0.6787635684013367, 0.7571194171905518, -0.7535576820373535, -0.7775406241416931, 0.5737480521202087, 0.016918674111366272, -0.00833230372518301, 0.11065074056386948, 0.9391293525695801, 0.6803441643714905, 0.475930392742157, -0.722450852394104, 0.6196770071983337, 2.482828378677368, -1.1418617963790894, -0.11717264354228973, 0.38161754608154297, -0.6667148470878601, 0.1913786083459854, -0.3506304919719696, -1.0131375789642334, 1.2860755920410156, 0.01835140399634838, 1.738896131515503, 0.5210656523704529, -0.2977263033390045, 0.4077796936035156, -1.3269256353378296, 0.32179680466651917, 0.5498568415641785, 0.4254864752292633, 0.5781699419021606, 0.09294283390045166, 0.8649116158485413, 1.339004635810852, -0.6668229103088379, 0.3380246162414551, -1.4161392450332642, -0.013458799570798874, -0.3774178922176361, -0.5124855041503906, -1.5432401895523071, -1.0599521398544312, 1.963101863861084, -0.5120925307273865, 1.6332848072052002, 0.38188230991363525, -0.4058718979358673, -0.9771853089332581, 1.7204376459121704, -0.030860692262649536, 0.05916382372379303, 0.1438983976840973, 0.3943270444869995, 0.3958125412464142, -0.2328309267759323, -1.9439387321472168, 0.11670269817113876, -0.4551205039024353, -0.7167751789093018, 0.636315643787384, -2.133115291595459, 0.8334031105041504, 0.40090250968933105, -0.08977263420820236, -1.02261483669281, 0.12254755198955536, 0.5952589511871338, -0.25221648812294006, -0.5213168859481812, -1.126343846321106, 0.512357771396637, -0.9695104956626892, -0.6963585019111633, 0.979097306728363, 1.7639763355255127, -1.1882553100585938, -0.8560973405838013, -1.2503371238708496, -1.0369501113891602, 0.052123937755823135, -0.3130965530872345, 1.201565146446228, 0.8599128127098083, -0.2786809206008911, 0.8275138139724731, -1.007059931755066, 0.28406843543052673, 0.08650866150856018, -1.0444016456604004, -2.0801126956939697, 0.14473628997802734, 0.17408069968223572, -1.0050204992294312, 0.18397095799446106, 0.40319034457206726, 1.2196499109268188, 0.1251763254404068, -0.6374293565750122, -1.3447155952453613, -1.5162575244903564, 0.42297300696372986, 0.4075782299041748, -0.33510300517082214, 0.2960548996925354, 0.4576954245567322, -1.193509817123413, 0.08998587727546692, -0.589549720287323, 1.8574093580245972, 0.29917415976524353, -0.9761300086975098, -0.10434781759977341, 0.05490747094154358, 0.42189908027648926, 0.18951916694641113, 0.06104075908660889, -0.693168580532074, 0.392095148563385, 0.053224820643663406, -0.09026210010051727, -0.28231775760650635, -1.6615753173828125, 0.14041222631931305, 0.1660652756690979, -0.46284157037734985, 0.7336542010307312, -0.24029012024402618, -0.16434124112129211, 0.5624998807907104, -0.011828669346868992, 0.3394750952720642, 0.607323169708252, -0.2233051359653473, 1.1149626970291138, -0.37342166900634766, -0.30804386734962463, 1.2175394296646118, 0.38087892532348633, -1.436299204826355, -0.39580368995666504, -0.7075758576393127, -1.4892311096191406, -0.22493799030780792, 0.22431103885173798, -0.7113643288612366, 1.668809413909912, 0.18364869058132172, -0.4436503052711487, 1.162872076034546, -0.1607288271188736, 1.7328697443008423, 0.17706044018268585, 0.8533727526664734, -0.10411521792411804, -0.9652842283248901, -0.9409885406494141, -0.8955080509185791, -0.5040464401245117, 0.281741738319397, 0.4935576021671295, 1.840733528137207, -0.09940583258867264, -0.3357298672199249, 0.31486448645591736, -1.0294430255889893, -0.7015800476074219, -0.9794296026229858, 0.46253108978271484, 1.6693476438522339, -0.5502282977104187, 0.7197680473327637, -1.0123404264450073, -0.5298395752906799, 0.5770400762557983, 1.5353275537490845, -1.9338022470474243, -0.09679566323757172, -1.0703835487365723, -0.4057789444923401, 0.8569103479385376, 0.07865104079246521, -0.09544260799884796, -0.22306568920612335, -0.5305353999137878, -1.260595679283142, -0.9496050477027893, -0.8725087642669678, 3.8003482818603516, 0.6100218892097473, -2.166876792907715, 0.07256082445383072, 0.5080543160438538, 0.02949114330112934, 2.208371877670288, -2.059810161590576, -0.2696495056152344, -0.03926663473248482, 0.25809648633003235, -0.2697162330150604, 2.2205018997192383, -0.33012086153030396, -0.6543459296226501, -0.02676493301987648, 1.110102891921997, 0.27454930543899536, -1.1074321269989014, 0.29622170329093933, -0.6686112880706787, -2.033999443054199, 1.4400126934051514, -1.7543835639953613, 1.4096500873565674, -0.30714666843414307, -2.3437931537628174, 0.9998873472213745, 0.6244693398475647, -0.19825047254562378, -1.5576117038726807, 0.03222263231873512, 2.5382800102233887, 1.7619428634643555, 0.5903549790382385, -1.5110887289047241, -0.8377024531364441, -0.4273889660835266, 0.4637937545776367, -1.1552627086639404, 1.2670693397521973, 1.2509500980377197, 0.40940165519714355, 0.773131787776947, -1.1620241403579712, -1.4716511964797974, 3.318169593811035, -0.4083637595176697, -0.0022009413223713636, 0.49424752593040466, 0.23967097699642181, 0.20398765802383423, 1.3099377155303955, 0.2639605700969696, -0.6952332258224487, 0.33059701323509216, -0.6314685344696045, 0.3716201186180115, 0.12319571524858475, 0.4640406668186188, -0.6292489767074585, 0.1105833575129509, -0.16181765496730804, 1.1160588264465332, -0.10065679252147675, -0.11882411688566208, -0.9520128965377808, -0.4396027624607086, 0.46750733256340027, 0.13091996312141418, 0.3119376301765442, 1.37665593624115, 1.1020023822784424, -0.06916007399559021, -1.1469725370407104, 0.9515698552131653, -0.7370468974113464, -1.3092070817947388, -1.139480471611023, 2.0632808208465576, -1.2105666399002075, 0.04657783731818199, 0.026759058237075806, 1.3491021394729614, 1.0412230491638184, 1.3447164297103882, 1.1985348463058472, 1.707123875617981, -0.334077388048172, -0.8461896181106567, -1.0999351739883423, -1.126097559928894, -2.2465808391571045, -0.43017876148223877, -0.05509168282151222, 0.22010762989521027, -0.18612022697925568, -0.39745405316352844, 0.8051470518112183, 0.21330220997333527, -0.10183069109916687, 0.561320424079895, 0.6864315867424011, -0.16403429210186005, 0.2927040159702301, -0.8949029445648193, 0.4527360498905182, 1.4047203063964844, 0.8698785305023193, 1.0287400484085083, 1.043891191482544, -0.6432573199272156, 0.13705292344093323, 0.13634933531284332, 1.663909912109375, -1.243958830833435, -0.6963000297546387, -0.2962092161178589, -0.44079211354255676, -0.2019873708486557, 1.2265713214874268, 0.8994871377944946, -0.23491716384887695, -1.143606424331665, 1.5564262866973877, -2.0420494079589844, 1.84080171585083, -0.35875385999679565, -0.3458709418773651, 1.2136863470077515, 0.2763444185256958, 1.4320179224014282, 2.349950075149536, 0.5136324763298035, 1.4506512880325317, 1.1352118253707886, 0.019205428659915924, 0.7575575709342957, 0.2516552209854126, 0.05091644451022148, -0.17411550879478455, 0.20387518405914307, 1.8606656789779663, -0.12606094777584076, -1.5028431415557861, 1.6538455486297607, -0.3523268699645996, -1.5993297100067139, 1.5451477766036987, 0.3647667467594147, -1.6069567203521729, 1.3623067140579224, 2.3283169269561768, -0.17786437273025513, -0.2005586177110672, 0.4641175866127014, 0.40368616580963135, 0.01626095362007618, -1.0798040628433228, -0.3709633946418762, 1.0769097805023193, 0.3838011920452118, -0.2098478525876999, -0.6097539663314819, 0.11603692919015884, 0.1209418997168541, 1.100906491279602, -0.38942939043045044, 3.1113083362579346, -0.33664897084236145, -0.9465925693511963, -1.2004129886627197, 0.07370597869157791, -0.40400391817092896, -1.4252315759658813, -0.48876136541366577, 1.6309272050857544, 0.25467994809150696, -0.24337022006511688, 0.05481690913438797, 0.12727032601833344, -1.9353121519088745, -0.325806587934494, 0.8911041617393494, 0.9905794858932495, 2.320909261703491, 1.2933160066604614, -1.5352933406829834, -0.20726485550403595, 1.400390625, 0.15428052842617035, 0.030351899564266205, -1.6477715969085693, -0.5198384523391724, -1.3196619749069214, -2.4133541584014893, 0.70064377784729, -0.09524461627006531, 0.3929910957813263, -1.008813738822937, 0.45119068026542664, 0.27225765585899353, 0.6480802297592163, 1.3538497686386108, -0.7592780590057373, 0.8662890195846558, 0.006741981953382492, -0.01750456541776657, 1.434643030166626, -0.69547039270401, 0.2941919267177582, 0.42955857515335083, 0.8532885313034058, 0.14613036811351776, -0.23202694952487946, 1.5176090002059937, 2.04832124710083, 0.19949784874916077, 0.7762572765350342, -0.36469507217407227, -1.0934722423553467, -1.9570564031600952, -0.6743214726448059, -0.4093533158302307, 0.5923896431922913, -0.06409022212028503, 1.0794023275375366, 0.3152056932449341, 1.0350998640060425, -2.260981559753418, -0.6950663924217224, -1.2907423973083496, -1.0961050987243652, 0.4224497675895691, 0.8917887210845947, 0.6337224841117859, -0.8921966552734375, 0.07031527906656265, -0.03753439337015152, -0.15819206833839417, 0.8165808320045471, -0.51762855052948, -0.7920003533363342, 1.6255192756652832, -0.5023669600486755, -1.2040939331054688, 0.5448388457298279, 0.055841803550720215, -0.38556697964668274, 0.25675976276397705, 1.6587783098220825, 0.6642423868179321, 1.1573175191879272, 0.5414268374443054, 0.9863709807395935, -2.3354647159576416, 0.4049968719482422, 0.021697960793972015, 0.14093118906021118, -0.3964856266975403, 0.4807227551937103, -1.397696614265442, 0.9308714866638184, -0.18224036693572998, -0.04251810908317566, -0.5240441560745239, 3.7216522693634033, 0.3974493145942688, -0.21702368557453156, 1.0000646114349365, 0.18683196604251862, 0.20186205208301544, 0.29336369037628174, -0.39334946870803833, -0.8432617783546448, 0.24045412242412567, -0.7673930525779724, -0.1022276058793068, -0.03979124128818512, -0.3351096510887146, -1.5025469064712524, -1.4173041582107544, 1.3417258262634277, -0.2477424144744873, 1.2010284662246704, -1.0975061655044556, 0.24608710408210754, -0.16506189107894897, -0.2915823757648468, 1.161791205406189, -1.558792233467102, -0.2229924201965332, -0.7499198913574219, 0.5804019570350647, 0.9904675483703613, 1.3768465518951416, -2.0047428607940674, -1.5527619123458862, 0.1563175618648529, -1.4789633750915527, -0.20772625505924225, 0.343241423368454, -0.1526682823896408, -1.4611148834228516, 0.9828019738197327, -0.2533552348613739, 1.2343266010284424, -1.483664631843567, 1.0007519721984863, -0.6784932613372803, -0.14000321924686432, -1.0106840133666992, 0.3381194770336151, -1.0046229362487793, 0.08815500140190125, 0.18115755915641785, 1.065810203552246, -0.5508335828781128, 0.1436273455619812, -0.5623446702957153, 0.36510327458381653, -0.7003388404846191, -0.603557288646698, 1.4034881591796875, 0.4821663498878479, -0.9038839340209961, 0.3829752504825592, -0.21732771396636963, 0.7096640467643738, -0.09927248954772949, 1.084985613822937, -1.810286283493042, 0.19962267577648163, 0.6980293989181519, -0.6633619070053101, 0.246721088886261, 0.7949457764625549, -0.5688223242759705, 0.4658605456352234, 1.0740703344345093, -1.352574110031128, 0.27028265595436096, -0.7187897562980652, -0.31598156690597534, -0.6587252020835876, 0.3001195192337036, -0.1537075787782669, -0.43505340814590454, -0.025874366983771324, -0.7652137875556946, -0.02246646024286747, -0.6191108226776123, 0.20385663211345673, 1.0671780109405518, -0.08662669360637665, 0.47359636425971985, -0.4938364028930664, 0.8618254661560059, 0.5007035136222839, 0.017130542546510696, 2.4408512115478516, -1.3409419059753418, 0.2697998583316803, -1.436592936515808, -0.46412619948387146, -1.4906433820724487, 0.17545630037784576, 1.2520697116851807], [0.9036588668823242, -1.5888874530792236, -1.9285194873809814, 1.3470743894577026, -1.3915218114852905, 1.3785887956619263, -1.7941420078277588, -2.124378204345703, 0.6299102306365967, 0.35574018955230713, -0.7591143250465393, 0.8035230040550232, -0.26036548614501953, -1.1853127479553223, 0.6321788430213928, -0.38942307233810425, -1.1266252994537354, -0.7724007964134216, 0.9416570663452148, 0.6781831383705139, 0.43675047159194946, 0.5363121032714844, 0.21501407027244568, 0.5021370053291321, -0.11599885672330856, -1.5309275388717651, -1.4637465476989746, 0.19192621111869812, 0.15908733010292053, 0.8930252194404602, 0.951287567615509, 2.3073153495788574, 0.06871207058429718, -1.8888790607452393, -0.06659521162509918, -0.5914784073829651, -0.21727336943149567, 1.0650315284729004, -0.49310678243637085, -0.2989619970321655, 0.39511406421661377, -0.34198740124702454, -0.3636837899684906, -0.12800350785255432, -0.398696631193161, 0.31688636541366577, 1.3787682056427002, -0.8442693948745728, 1.0829477310180664, -0.9657669067382812, -0.020988741889595985, -0.6045415997505188, 0.5429072380065918, 0.49400824308395386, 1.2372596263885498, 0.003394750412553549, 0.6707777380943298, -0.44572311639785767, -0.9455073475837708, -0.04660472273826599, -1.0436848402023315, -0.14811600744724274, 1.6362148523330688, 0.2890819311141968, 0.7896543741226196, 0.27986931800842285, 0.1681281328201294, 1.1743175983428955, -0.7290623784065247, 0.8320189118385315, -0.003327522426843643, -0.23206812143325806, 0.5060052275657654, -0.029444150626659393, -0.3134395182132721, -0.25013771653175354, 1.0121662616729736, -0.42826417088508606, 2.6721694469451904, -0.6418413519859314, 1.5487334728240967, 0.5710009336471558, -1.1313443183898926, 0.7586803436279297, -0.8717260360717773, -0.6601141095161438, -1.5543690919876099, -0.20478302240371704, -0.8600619435310364, -0.6060272455215454, 1.6130307912826538, 0.2942620515823364, -2.6593539714813232, 0.9349671602249146, -1.6279516220092773, -0.9634923338890076, 0.3022538721561432, 1.4778324365615845, 0.035981930792331696, -0.16928574442863464, 1.2736561298370361, -0.8423643112182617, -0.21736910939216614, -2.306239604949951, -0.289339542388916, 0.3275969922542572, -2.506463050842285, 1.1628994941711426, 0.5919050574302673, 1.130332589149475, -0.6722455620765686, 0.5431542992591858, -1.259666085243225, 0.4233037829399109, -0.8691775798797607, -0.4052208662033081, 0.0176599882543087, 0.1618214249610901, -0.6442613005638123, 0.3203182816505432, -0.9960491061210632, 1.4119176864624023, -0.1610001176595688, 1.5214065313339233, 0.080290287733078, 0.08027784526348114, 0.8486279249191284, 1.8145078420639038, -1.6943126916885376, 1.1393777132034302, -1.1172075271606445, 0.9925498366355896, -0.09674709290266037, 0.41402560472488403, -0.5245489478111267, -0.1479407101869583, -0.005424591712653637, 0.8406693935394287, -1.5177603960037231, 0.7708887457847595, -1.7867263555526733, 0.8333988189697266, 1.2203577756881714, 0.6117874383926392, -1.018965244293213, 1.0708515644073486, 1.2326465845108032, 0.9776036739349365, 0.8856495022773743, 0.36742308735847473, -0.1106710433959961, -1.3600624799728394, -1.091233491897583, -0.6460998058319092, -1.5215109586715698, 0.566892147064209, -1.1916664838790894, 0.1833745837211609, 0.3923068046569824, -0.4129306375980377, 0.3916967213153839, -1.0726600885391235, 1.3711930513381958, -0.3856061100959778, 0.6509949564933777, 0.6417996287345886, -1.8329477310180664, 1.8159483671188354, -1.5071148872375488, -1.061873435974121, 0.7039888501167297, 0.14060238003730774, 1.698472261428833, 1.3024187088012695, -1.4927258491516113, -1.2310373783111572, -0.6111409664154053, 0.3421993851661682, 0.38722750544548035, -0.6483703255653381, -1.8164715766906738, 0.9183388352394104, -1.491875410079956, -0.37450656294822693, -0.6240943670272827, 1.0301212072372437, 0.7236979603767395, 0.03389577567577362, -0.806147575378418, -0.37796998023986816, 1.1463119983673096, -0.1690196394920349, -0.7764325737953186, 2.7642345428466797, -0.4947487711906433, -1.9395451545715332, -0.8256214261054993, 1.043555736541748, 0.6915600299835205, 0.3603551983833313, -1.3742915391921997, -0.573644757270813, 0.6502954959869385, 0.1159089058637619, -0.8962484002113342, 2.4210445880889893, 0.4540018141269684, -0.05736609175801277, 0.7468110918998718, 0.5772017240524292, 2.8029234409332275, -0.11245584487915039, 1.1512395143508911, 1.538405179977417, 0.9032377004623413, -1.1846634149551392, 0.742350697517395, 0.6301243305206299, 0.13302329182624817, 0.3687766194343567, -0.9469812512397766, 0.7476232051849365, 0.6515592336654663, -0.9765427708625793, 0.18927760422229767, 1.5901373624801636, -0.21859435737133026, 1.3105273246765137, -0.17245171964168549, 1.6259114742279053, 0.6791009306907654, -0.7517105340957642, -0.0035963833797723055, 2.69138765335083, 0.8253382444381714, 0.2741926908493042, 0.49377602338790894, 1.2615418434143066, -0.2476031631231308, 0.6986467838287354, 0.14834225177764893, -0.08142402023077011, -1.1179314851760864, 0.8469803333282471, 1.8347564935684204, -0.44109058380126953, -1.6223740577697754, -0.7177110314369202, 0.033594608306884766, 0.20438717305660248, 1.311417579650879, -0.6898694038391113, 0.6930667757987976, -0.20164091885089874, 0.16748259961605072, 0.16517767310142517, 0.5860956311225891, -0.5382469892501831, 0.913986325263977, 0.8313105702400208, 2.232126474380493, 1.2850080728530884, 2.869393825531006, -0.06458848714828491, 0.7693031430244446, 1.1331621408462524, -0.06537951529026031, 1.5616077184677124, 2.1483399868011475, -0.14665308594703674, 1.8210594654083252, 0.1557425856590271, 1.070300817489624, 1.6639777421951294, 1.5244700908660889, 0.5023601651191711, -0.3439900577068329, 0.8868879675865173, 0.3147096335887909, -0.28104570508003235, 0.667298436164856, 0.1531972587108612, 0.2654392123222351, 0.1755634993314743, 0.12535250186920166, -0.48543065786361694, 0.6210678219795227, 0.21079957485198975, -0.05353366956114769, 0.20668993890285492, -1.7167173624038696, 0.01967381313443184, 0.6493276357650757, -0.9994820356369019, -0.3461967408657074, -1.0952746868133545, -0.7466739416122437, -0.2127096951007843, 1.4074249267578125, 1.3182190656661987, -0.272295206785202, -0.3139339089393616, 1.4221006631851196, -0.17333495616912842, 0.7885709404945374, 0.7715638875961304, -0.8484338521957397, -1.2502686977386475, 0.9666056632995605, 0.552502453327179, 1.1513506174087524, -0.11425960808992386, 1.416539192199707, 1.4491450786590576, 0.9584714770317078, 2.2422447204589844, 1.3404618501663208, 0.8930712342262268, 1.7720849514007568, 1.5344741344451904, 0.26654934883117676, 0.7516799569129944, -0.5715327262878418, 0.005447952542454004, -0.2214292734861374, 0.09388186782598495, -0.39027664065361023, -0.16836047172546387, 0.043469905853271484, -0.6434157490730286, -1.8204503059387207, 0.023595282807946205, 0.27607810497283936, 0.8566939234733582, -1.8004701137542725, 1.8346660137176514, -0.5433927178382874, 0.42444971203804016, -0.020138563588261604, -0.7584554553031921, -0.44210904836654663, 0.7134912610054016, 0.20370523631572723, 0.05692966282367706, -0.6604208946228027, 0.9702022075653076, -0.8246042132377625, -0.01470300555229187, 0.022792866453528404, 0.9268598556518555, -0.02314598113298416, 0.21562135219573975, -0.8293436765670776, 0.40906354784965515, -1.4102022647857666, -0.3775383532047272, 0.5114035606384277, -0.5951177477836609, -0.5410794615745544, -1.0591437816619873, -0.6777790784835815, 0.24046383798122406, 0.07211726158857346, 1.0008807182312012, -0.9045339822769165, -0.16402342915534973, 0.6675747632980347, -0.5891861319541931, 1.0504429340362549, 0.21775192022323608, -1.0189038515090942, 2.7322914600372314, 0.2226516157388687, 0.5035730004310608, -1.0034139156341553, -0.41095274686813354, 0.423329621553421, -2.5524306297302246, -0.9486529231071472, 0.031125206500291824, -0.9081177711486816, -0.22296519577503204, -2.1294755935668945, -0.27441197633743286, -0.0246187262237072, 0.6580672860145569, 1.5780986547470093, 1.5611228942871094, 0.5595727562904358, -0.9966011643409729, -0.9371532201766968, -0.18412566184997559, -0.3296548128128052, -0.8610215187072754, -0.15262985229492188, -0.002744509605690837, -1.9404306411743164, 0.5980543494224548, 0.5583520531654358, 1.3254684209823608, 0.6258626580238342, 0.6574931740760803, -0.25452908873558044, -1.621965765953064, 0.6960199475288391, 0.6398951411247253, -1.4675079584121704, -0.3720203638076782, 2.016773223876953, -1.771305799484253, 0.3170537054538727, -0.5626342296600342, -0.466734379529953, -0.6276246309280396, -0.43385523557662964, 0.2429255098104477, -1.988157868385315, 0.5579246878623962, 0.3881782293319702, 2.834758996963501, -0.2922946810722351, 1.3451889753341675, -0.8680964708328247, -0.24462704360485077, -0.756338357925415, -0.27454930543899536, -1.7906500101089478, 1.541347622871399, -2.704002618789673, -1.7598174810409546, 0.4201270043849945, -1.175889492034912, -1.7165911197662354, 0.24943652749061584, -0.586663544178009, 0.7792288064956665, 0.5190627574920654, 0.7069650292396545, -0.5624476671218872, 1.6794869899749756, -1.3105645179748535, 0.37772858142852783, -0.0809759795665741, 0.66627037525177, -2.049513101577759, -1.1915210485458374, -0.03736274316906929, -0.10010500997304916, 0.046502914279699326, 1.3580011129379272, -1.101979374885559, -0.21867822110652924, 1.2962490320205688, -1.3686347007751465, 0.18051989376544952, 0.723103940486908, 0.26052340865135193, 0.8688074350357056, 0.7118051052093506, -0.7251303791999817, 0.5596030354499817, -1.6245763301849365, -0.036112017929553986, -0.3343990445137024, -0.6211650967597961, -0.8920360803604126, -0.9504455327987671, -0.44525033235549927, -0.539607584476471, 1.8507025241851807, 2.3042232990264893, 1.1781662702560425, -0.3056356608867645, 1.1689106225967407, -2.089488983154297, -0.1475113481283188, -0.7344890236854553, -0.4733157753944397, 1.7076059579849243, 0.372769832611084, 0.2042110413312912, -1.1056891679763794, -0.25881245732307434, -0.5570569634437561, 0.09237249195575714, -0.8572901487350464, 0.5900745987892151, 0.7981362342834473, 0.7610220909118652, -0.538600504398346, -0.16467295587062836, 1.1730539798736572, -1.6383036375045776, 1.3466808795928955, 0.23419414460659027, 0.6176214814186096, -0.2553524971008301, 1.3696409463882446, 0.16523753106594086, 0.08277427405118942, 1.371130108833313, 0.3991340100765228, 0.6699989438056946, 0.32536646723747253, 0.7325617671012878, -0.5230588912963867, -0.5058645606040955, -0.6754435896873474, -1.0234700441360474, -0.060242511332035065, 1.7999697923660278, -1.2363466024398804, 0.16276757419109344, 1.4618405103683472, 0.08720573782920837, 0.5302053093910217, -1.1271857023239136, 1.0472500324249268, -0.5318524837493896, 0.8031870722770691, -1.8103563785552979, 0.16759875416755676, -0.2482178658246994, 0.4822155833244324, -0.2894130051136017, 0.4842134118080139, 0.14979512989521027, -1.212339162826538, 0.10675473511219025, -0.2488800585269928, 1.372568964958191, 0.5416063070297241, -0.11101718246936798, 0.018054990097880363, -0.09593991190195084, 0.03362058103084564, -2.748762369155884, 0.5434598326683044, 0.17543072998523712, 1.467644214630127, 0.09093324095010757, -0.10538505017757416, 2.0723516941070557, 2.2720205783843994, -0.5082132816314697, -0.13089138269424438, -0.5553987622261047, -2.216752529144287, -0.42180728912353516, -0.965050220489502, -0.8192217350006104, -0.30367743968963623, 1.718222737312317, 0.8147212266921997, -0.17999163269996643, -1.1338053941726685, -1.3190487623214722, -0.6716441512107849, -0.5801778435707092, 0.06909149885177612, -0.1885337382555008, -0.47873538732528687, 0.5240400433540344, -0.0914105772972107, -0.7656016945838928, 0.9606665372848511, -1.0494122505187988, -0.6798977851867676, 0.06634978204965591, -0.3426814675331116, -0.9387184381484985, -1.4677536487579346, 0.29180485010147095, 1.8039922714233398, 1.603299856185913, 0.03122146800160408, -0.16644369065761566, -0.9852171540260315, 0.11933454126119614, 0.3386208713054657, -0.8787031173706055, -0.29833704233169556, 2.0550925731658936, 1.2893892526626587, -0.20218674838542938, -0.6298918128013611, -0.8826913833618164, -1.7742136716842651, -1.6868900060653687, 1.480085015296936, -0.21545833349227905, -0.9049406051635742, 0.05960327759385109, -0.21736101806163788, -0.09137997776269913, -0.8178044557571411, 0.04048458859324455, 1.0951107740402222, -1.3783608675003052, 0.5574734210968018, -0.03179216757416725, -0.5616782307624817, 1.3876465559005737, -0.2554040253162384, 0.47342804074287415, 1.3260858058929443, 0.657338559627533, 0.07464513927698135, 0.7808682918548584, -0.1119765192270279, -0.12273788452148438, 0.40706929564476013, 1.263953447341919, 0.6593706011772156, -1.381992220878601, -0.6224756836891174, -0.4035665690898895, -0.18389596045017242, -1.1512514352798462, 0.13083550333976746, 3.106274127960205, -0.02282801829278469, -1.5352810621261597, 1.8314636945724487, -1.2308406829833984, 0.887763500213623, 0.04506256431341171, 1.3211634159088135, -1.2640113830566406, -0.7437108159065247, 1.207801103591919, -0.025034138932824135, -0.19179630279541016, -1.4505531787872314, 1.9143975973129272, -0.28872615098953247, -1.9970070123672485, -1.4547615051269531, -0.525558352470398, 0.643261194229126, -0.6554685235023499, -0.7130733728408813, 0.22077339887619019, -0.04815584048628807, 1.3550785779953003, 0.07611491531133652, -1.3064767122268677, -0.48560160398483276, -0.014022104442119598, 0.3171524107456207, -0.667368471622467, -0.29965800046920776, -1.4464564323425293, 0.16141386330127716, -1.6121249198913574, 0.6298354268074036, -0.804936408996582, 1.0985238552093506, -1.0232666730880737, 0.2618054449558258, -1.5259010791778564, 2.0702736377716064, 0.07529095560312271, -0.8389981985092163, -0.3908393085002899, -0.27845990657806396, -0.24616631865501404, -0.08531153947114944, 0.28289854526519775, 0.2943810522556305, -0.0698321983218193, -0.9510734677314758, -0.08342870324850082, -0.7724294066429138, -0.5366160273551941, -1.9020812511444092, -2.2698755264282227, 0.6855624318122864, -0.9405617117881775, 0.22120201587677002, -1.1278767585754395, 1.6653366088867188, 1.915920376777649, -0.7146894931793213, 0.09134690463542938, 1.0704355239868164, 0.9581241607666016, -0.19390740990638733, 1.7001017332077026, -0.702022135257721, 0.12668275833129883, -0.3589608669281006, -0.33986198902130127, 0.6315143704414368, 0.7888407111167908, 0.12706024944782257, -1.4427047967910767, -0.8460212349891663, -0.9349232316017151, 2.381181478500366, 1.1619575023651123, -0.02934863790869713, -2.402967691421509, -1.4347939491271973, 0.11050337553024292, -1.1268260478973389, -0.9096392393112183, -0.1487313210964203, 1.70518159866333, 0.5281351208686829, -0.687119722366333, -0.18113113939762115, 1.0731252431869507, -1.9935495853424072, -1.2419761419296265, -0.3254266679286957, -0.09849853813648224, -0.6768341660499573, 0.31543368101119995, -1.5161938667297363, -0.5864647030830383, 0.43692898750305176, 0.26062729954719543, 0.2860909104347229, -0.48159754276275635, -0.5565077066421509, -1.2574176788330078, 1.0261470079421997, 0.4970793128013611, 0.5718300342559814, 0.8133314847946167, 0.8598084449768066, -0.8974094986915588, 0.556034505367279, 0.27743932604789734, -0.1282528042793274, -1.3102529048919678, -1.5031137466430664, 0.23248425126075745, -0.31489095091819763, -1.243208646774292, 0.35098233819007874, 0.23775798082351685, -0.4458203613758087, -0.20999754965305328, 0.5026667714118958, 0.145949125289917, -0.10430163890123367, -0.9820636510848999, -0.668704092502594, 1.199565052986145, -1.5097628831863403, -0.42097488045692444, 0.4522656500339508, 0.760771632194519, -0.988739013671875, 0.5129579305648804, -1.5303887128829956, 0.3013902008533478, -0.43661144375801086, 0.4211694896221161, -0.3200463652610779, -0.030616816133260727, -0.3546110689640045, 0.5684866309165955, -0.9868074655532837, 0.5883431434631348, 1.2825267314910889, 0.03895760327577591, 0.6146023869514465, -0.2863888740539551, 0.20028360188007355, -0.15296806395053864, -1.0831605195999146, 0.8210639953613281, 0.2608965337276459, 0.34350037574768066, -0.31496351957321167, -0.5867512226104736, -0.8676077127456665, -3.154045581817627, -0.49829238653182983, -0.5977057218551636, 0.9456360936164856, -0.7146164774894714, 1.7936296463012695, 0.2604414224624634, 0.15936246514320374, 0.9540941715240479, 0.029016098007559776, 3.0476722717285156, 0.8118611574172974, 1.3560410737991333, 0.2881174087524414, 0.8604960441589355, -0.12209238857030869, -0.4929417371749878, -0.24478201568126678, 0.7995343804359436, -0.7499207258224487, 0.1297154426574707, 1.4617255926132202, -1.1061816215515137, -0.4841232895851135, -1.7032588720321655, 0.47085610032081604, -1.2154957056045532, -1.182371735572815, 0.2719530165195465, 1.4446734189987183, -1.0549800395965576, 0.14210212230682373, 0.6982207298278809, 0.4225163757801056, -1.4739750623703003, 0.7501347661018372, 0.4784190356731415, -0.3853563070297241, -1.70337975025177, -0.2244778871536255, -0.29635488986968994, -0.8828981518745422, -0.6394981741905212, -0.9200211763381958, 0.4060681164264679, -0.26778295636177063, 1.6844005584716797, 0.4227997064590454, -1.530648946762085, 2.1610419750213623, -0.325314462184906, 0.6187963485717773, 1.3431321382522583, -0.015424898825585842, 0.13365954160690308, -0.7872313857078552, -0.5916551947593689, -1.1377155780792236, 0.7765226364135742, 0.0936751589179039, 0.6842988729476929, 0.19590391218662262, 0.10416384786367416, -0.7525662183761597, 0.5708636045455933, 0.9657548666000366, -0.05606317147612572, -1.0897027254104614, -1.1123902797698975, 0.1683281511068344, 0.545050323009491, -0.8892166018486023, -1.3331413269042969, -1.6775133609771729, 0.6516513228416443, 0.8761854767799377, 0.4914509654045105, 0.2896213233470917, -0.6898297667503357, 0.1034567728638649, -1.0322812795639038, -1.0583428144454956, 1.1423121690750122, 0.3736167252063751, 1.557106375694275, -0.2796081602573395, 0.3985023498535156, 0.8219485878944397, 1.5628292560577393, 1.153417944908142, -0.8684918284416199, -2.436018705368042, 2.137549638748169, 1.073714256286621, 0.11748873442411423, -0.5254032015800476, -0.8270267248153687, -0.6486894488334656, 0.6124622821807861, 0.22776813805103302, -0.9236815571784973, 0.16819100081920624, -0.13301977515220642, -0.5909166932106018, -0.511260449886322, -0.03716329112648964, 0.4054526090621948, -0.585756242275238, -0.4454084634780884, -1.0685615539550781, 0.19082121551036835, -0.20174947381019592, 0.16561122238636017, 0.6679353713989258, 0.36127200722694397, -0.11580772697925568, -0.6596484780311584, 3.0422022342681885, 1.0614256858825684, 0.454407662153244, 0.9299352765083313, -0.033228129148483276, 0.3433076739311218, -1.3851553201675415, 0.6179991364479065, 0.7458367347717285, -0.7545564770698547, -0.31143099069595337, -1.0477474927902222, -1.5155433416366577, -1.4824000597000122, 0.019929172471165657, -0.4902327060699463, 0.36911842226982117, -1.081774115562439, -0.4190208613872528, -0.1274627447128296, 1.4719539880752563, 1.0360702276229858, 2.2534289360046387, -0.943915605545044, 0.4316750168800354, -0.2469426691532135, -0.09451039135456085, -1.225625991821289, -0.7535144686698914, 1.2622723579406738, -1.1961064338684082, 1.2369929552078247, -1.4683362245559692, -1.0139051675796509, 1.1955382823944092, 0.020654261112213135, 0.12205033749341965, -1.7090890407562256, -0.11596810817718506, 0.03348485007882118, -0.5927942991256714, -1.2558139562606812, -0.42736122012138367, -0.8982849717140198, 0.24556121230125427, 0.8598049283027649, 0.2707515060901642, -0.7564031481742859, 0.3047932982444763, -1.196286916732788, 0.5226287841796875, 0.47977668046951294, 0.5784066319465637, 0.09544141590595245, 0.8432705402374268, 1.033862829208374, 0.38968995213508606, -0.47193825244903564, -0.7463660836219788, -0.35438671708106995, -0.0772157832980156, -1.5393216609954834, -0.24887466430664062, -1.2439535856246948, 0.48574191331863403, 0.7202431559562683, -0.8570888042449951, 0.21280895173549652, -0.4622322618961334, -2.60500431060791, -0.8159881830215454, 0.2714032232761383, -0.5436106324195862, 0.3350408673286438, 1.4790434837341309, -1.2718329429626465, 0.6164941787719727, -0.054958101361989975, -0.5938593745231628, -0.5082750916481018, 0.6749882102012634, -0.2961309254169464, 0.12087643891572952, -0.0910954475402832, -0.9250027537345886, -0.2528800964355469, -0.08945678919553757, -1.0242756605148315, 0.6635145545005798, -1.016503095626831, -1.1344573497772217, 0.40419092774391174, -0.8811866641044617, -1.0168660879135132, 0.23998230695724487, 0.29911041259765625, -1.0192945003509521, -0.9452931880950928, -1.9408907890319824, -1.7691844701766968, -0.7273383736610413, -1.3792058229446411, -0.6704613566398621, 0.897955060005188, 0.45467597246170044, -1.2454960346221924, 0.8795604109764099, -0.022855741903185844, -0.8427717685699463, -1.3231587409973145, 0.39754238724708557, 1.7181923389434814, 0.44652289152145386, 0.3080393373966217, -1.5611748695373535, 0.012045652605593204, -0.7031537294387817, -0.33659279346466064, -0.8565065860748291, 0.7549777030944824, -0.4667191207408905, 1.215948224067688, 1.3807640075683594, 0.09800774604082108, 0.3631947338581085, -1.3538689613342285, 0.849848747253418], [-0.3491378426551819, -0.6596691608428955, -2.6463656425476074, 0.14772452414035797, 2.317875862121582, -1.38571035861969, -0.6651020050048828, -0.00910926889628172, -0.8173366189002991, 1.0933302640914917, 1.2141772508621216, 0.6182357668876648, -2.0969431400299072, -0.37971773743629456, -0.29500722885131836, 1.3960703611373901, -0.4780277609825134, -0.17854425311088562, 0.7614944577217102, 1.3149303197860718, -0.1377914994955063, 0.5353020429611206, 0.8865334987640381, 0.3711478114128113, -0.21339862048625946, -0.20649462938308716, 0.1403399258852005, -0.47782188653945923, 0.050945162773132324, 1.736487865447998, 0.8085874915122986, 1.9553841352462769, 0.9364545345306396, -0.03350473567843437, -0.2687865197658539, 0.5692543387413025, 1.2756671905517578, 0.9632827043533325, 0.852598249912262, 0.13451015949249268, -0.03601802885532379, -0.36917075514793396, -0.20724479854106903, -0.736149251461029, -1.1985044479370117, 0.20146067440509796, 0.08530410379171371, -1.6193938255310059, -0.11615162342786789, -1.4103807210922241, -0.3179512321949005, 1.4057493209838867, -1.4624067544937134, 0.44825366139411926, 0.17098137736320496, 0.46955323219299316, 0.8736284375190735, 0.06196630001068115, 1.626710295677185, 0.07022660225629807, 0.11627581715583801, -0.4614657461643219, -2.1051323413848877, -0.5849195122718811, 0.16240978240966797, 0.9131447076797485, -0.4426136910915375, -0.06591890007257462, -1.362938642501831, -2.0422167778015137, -0.0550130270421505, 0.5715712904930115, 1.3761659860610962, 0.08093728870153427, 0.8702451586723328, -0.8392029404640198, 0.11306183785200119, 1.9945396184921265, 0.1685205101966858, -0.6548691987991333, -1.9148231744766235, -0.004890547599643469, -0.24956917762756348, 0.4832019805908203, 0.7682269811630249, -0.6630516648292542, 0.3963978886604309, -0.8038477301597595, 0.10975559055805206, 0.19518673419952393, 0.1429053097963333, 0.02098904922604561, -0.3819018304347992, -1.58113694190979, 1.7518287897109985, -0.3299561142921448, 0.0874156728386879, -0.2874838709831238, 0.544348418712616, 0.26655182242393494, -1.3425743579864502, -0.6436911821365356, 0.5589920282363892, -0.5889270901679993, -0.23024633526802063, 0.25883913040161133, 0.5810925960540771, 0.14797049760818481, 0.5866243243217468, 1.97256600856781, -0.8358233571052551, 0.2988826036453247, 1.6637465953826904, 0.003983266651630402, -2.576014757156372, 0.860958993434906, 0.6039172410964966, 1.272375226020813, 0.7761233448982239, -1.305599570274353, -0.14753015339374542, 1.3586512804031372, -0.39456140995025635, -0.3038371205329895, -0.13933293521404266, -0.7156115770339966, 0.7404966354370117, 1.0958970785140991, -0.7012501358985901, 0.5957464575767517, 0.3631994128227234, -0.06900177896022797, -1.1113011837005615, 1.2412288188934326, -0.35570210218429565, -0.5241766571998596, 0.6772106289863586, -0.1250249445438385, -0.30634546279907227, -0.8922935128211975, 0.6993597149848938, -0.9182291626930237, 0.5266493558883667, 0.8853927850723267, 0.2525242269039154, 1.0677237510681152, 0.6185368299484253, -0.3991643190383911, 0.16108585894107819, 0.2177320420742035, 1.1013911962509155, 0.4429701566696167, 0.3433071970939636, -1.394798994064331, 1.0661720037460327, -0.265106201171875, 1.560268521308899, -1.1412253379821777, -0.1280989795923233, 0.8750074505805969, 1.4292752742767334, 0.07104063779115677, -0.5277461409568787, 0.7110188007354736, -0.902564525604248, 0.9443792700767517, 0.4353930354118347, 0.8255749344825745, 1.2973252534866333, -0.16509979963302612, 1.2253525257110596, -0.38382795453071594, -1.4600111246109009, 1.1632202863693237, 1.5888348817825317, 0.8133010268211365, -0.9062936305999756, -0.778204619884491, -2.223766565322876, -0.1187003031373024, 0.6780725717544556, -0.05856046453118324, 1.3805932998657227, 0.7151660323143005, -0.23654954135417938, 0.017561981454491615, -0.32013264298439026, -0.766076385974884, 0.6297977566719055, -1.108673334121704, -1.8297358751296997, -0.3174494504928589, 0.6403737664222717, -1.7734099626541138, 0.12989959120750427, 0.4575517177581787, 0.9744735360145569, -0.275898814201355, -0.6819660067558289, 0.3826243281364441, -0.47205308079719543, -0.6019580960273743, 0.7369750142097473, 0.19153992831707, 1.7058191299438477, 1.8641152381896973, -0.43153613805770874, 0.14125920832157135, -0.4884728491306305, 0.36694058775901794, 0.3418474495410919, 1.9061630964279175, 1.118505597114563, 0.6303203105926514, -0.9792625904083252, -0.4811042249202728, 0.9603283405303955, -0.8123797178268433, 0.523248553276062, 0.012634245678782463, -0.9908086061477661, -1.1929652690887451, -2.584538221359253, -0.8971714377403259, -0.5992768406867981, -0.17701780796051025, -0.2065904289484024, 0.33411821722984314, -1.702088713645935, -1.2783664464950562, -0.5126240253448486, -0.44439271092414856, -1.3698934316635132, 0.01314549520611763, 0.33691465854644775, 1.5529863834381104, 1.0298066139221191, -0.4174031615257263, 0.29129087924957275, 0.28156524896621704, -1.0475553274154663, -0.5236881971359253, 0.6988220810890198, -0.48679912090301514, 0.09546422958374023, 0.9016754627227783, -1.0415270328521729, -0.9612129330635071, -0.3386741876602173, 1.1173971891403198, 0.5672551393508911, 1.3815722465515137, -1.1024845838546753, 2.048673152923584, 0.6514110565185547, -1.2158100605010986, 0.6620144248008728, 0.3961166441440582, 0.6271262168884277, 0.44508570432662964, 0.06068175286054611, -0.28073132038116455, 0.6185632944107056, 0.09148109704256058, -0.06427743285894394, 0.1602364331483841, -0.41613486409187317, -1.6372296810150146, 1.5917713642120361, 0.9871067404747009, 0.26924341917037964, -0.269288569688797, -0.3941057324409485, 1.141906499862671, 3.000084638595581, 0.07131759077310562, -0.8724514842033386, -0.38574081659317017, 0.3144088089466095, 0.10398789495229721, -0.30152416229248047, 0.13261403143405914, -0.053470175713300705, 0.6568014621734619, -0.7913421988487244, -0.21575525403022766, -0.6652323007583618, -0.628450334072113, 1.9757685661315918, 1.069344401359558, -0.8272078633308411, 1.3483843803405762, -1.3179166316986084, 0.23695597052574158, -1.157921314239502, -0.5737284421920776, 1.9348758459091187, 0.41175881028175354, -0.0334615521132946, 1.3444788455963135, -1.1631211042404175, -0.5911155939102173, 0.4450811743736267, -0.13247103989124298, 0.5487357378005981, 0.47268161177635193, -0.36369359493255615, -0.24739986658096313, 0.41797247529029846, 0.17293420433998108, 0.6568921208381653, 2.487499713897705, 0.22086547315120697, -0.07496367394924164, 0.19446039199829102, -0.1858193427324295, -0.7426666617393494, 0.5405423641204834, 0.38003599643707275, 0.555732011795044, -0.5716786980628967, -1.1324189901351929, -0.34578898549079895, 0.3655771017074585, 1.1993014812469482, -1.7153019905090332, 0.27494072914123535, -0.3256258964538574, 0.9829490780830383, -1.4221585988998413, 1.8871594667434692, -1.378572702407837, 0.10056116431951523, -0.6114171743392944, 0.2936075031757355, -0.03338261693716049, -0.4543990194797516, -0.4163799583911896, 0.049352411180734634, 0.5079589486122131, -1.2437257766723633, -0.32269173860549927, 0.3936511278152466, -0.13309186697006226, 0.10498806089162827, 0.4097592532634735, 0.8757338523864746, -0.8149154186248779, -1.3416000604629517, -1.7667343616485596, 0.68882155418396, 0.04243451729416847, 0.4752575159072876, -0.7417720556259155, 0.05760900676250458, -0.36307501792907715, -0.27845606207847595, -1.4813015460968018, -0.6274358034133911, -1.34101140499115, 0.22589276731014252, 0.03887154534459114, 1.8394594192504883, -0.5804028511047363, 0.05065983161330223, -1.9253251552581787, -0.7870387434959412, 0.7860331535339355, -2.266223669052124, 0.2776795029640198, -1.3355354070663452, -0.05886173993349075, -0.15940147638320923, 0.25583064556121826, 0.5567936301231384, -1.0107005834579468, -0.28644120693206787, 1.1487560272216797, 0.9758739471435547, 0.6470557451248169, -0.6170386672019958, -0.1543484777212143, 0.4244445264339447, -0.41459527611732483, 0.5585029721260071, 1.5681496858596802, -0.00619394751265645, -0.5301711559295654, -0.3730158507823944, -1.0937951803207397, -0.2958681881427765, -1.5568071603775024, -0.04116664081811905, 0.07364016026258469, -0.03291667252779007, -1.450191617012024, 1.3840323686599731, 0.8360682129859924, -1.0844007730484009, -0.5765161514282227, -2.2820069789886475, -1.0015432834625244, -0.8586882948875427, 0.7199274897575378, 1.2515835762023926, 0.6138601303100586, 0.6164531707763672, -1.5668447017669678, -1.289527416229248, 1.6768072843551636, 1.1096549034118652, -2.010770082473755, 0.3529479503631592, -0.3101148307323456, 0.984399139881134, 0.14467526972293854, 2.7912721633911133, -1.1253201961517334, 0.1462610960006714, -0.5797193050384521, 0.6248574256896973, -1.495185136795044, -0.31938332319259644, -1.7999484539031982, -1.28633713722229, -0.6525159478187561, -0.059342678636312485, -1.4282723665237427, -0.5479695796966553, 1.450070858001709, -1.8716329336166382, 0.9665717482566833, 0.4485408663749695, 1.5208786725997925, -0.2580670416355133, 0.2913033962249756, -0.790982723236084, 0.06932523101568222, 0.6878744959831238, 0.4673570692539215, -1.5308862924575806, -2.3513925075531006, -0.26578307151794434, -1.3802236318588257, -0.47785404324531555, 0.2991214096546173, 0.8333594799041748, -0.5921840667724609, -0.25063198804855347, -0.8645080924034119, 1.856789231300354, 3.852513551712036, -0.07568877190351486, -0.5816808342933655, 0.6988176703453064, -0.01860673911869526, 0.5662837624549866, -2.301844358444214, -1.7743619680404663, -2.6731231212615967, -0.09525786340236664, -1.9983023405075073, 0.9268668293952942, -0.10454103350639343, -1.0117853879928589, 0.5512408018112183, -0.9678599238395691, -0.22400914132595062, 1.3374402523040771, 0.5445936918258667, -0.3256741166114807, 0.06505293399095535, -0.10272027552127838, -1.7036242485046387, 1.9072881937026978, -0.1083986833691597, -0.9223403334617615, -0.8248145580291748, -0.7752200961112976, 0.5159149169921875, 2.3856723308563232, 0.015236719511449337, 0.34955894947052, -0.15095093846321106, -0.8256908059120178, -0.06621401757001877, -0.9088749289512634, 0.5297180414199829, 1.1999555826187134, -1.821362018585205, -0.5526542663574219, -0.5018882751464844, -0.022422561421990395, -0.4090943932533264, 0.156746506690979, -0.921006977558136, 1.612919569015503, -1.4632749557495117, 1.4138424396514893, -0.14966505765914917, 0.6740710139274597, 0.40680789947509766, -0.34766897559165955, -0.9808499217033386, 0.3900216817855835, -1.1426560878753662, 0.05995385721325874, 1.9204169511795044, 0.17417608201503754, 0.45513105392456055, 1.26680326461792, 0.9436748027801514, 0.9704939723014832, 0.2339455783367157, 0.5884144306182861, -0.18461690843105316, 1.441465139389038, 1.2642754316329956, -0.2630169093608856, -1.2351269721984863, -2.511186122894287, -0.5501472353935242, 0.044699810445308685, 1.5938583612442017, 0.6226852536201477, 1.0465129613876343, 0.5153312683105469, -1.589961290359497, -0.9193961024284363, -1.2044308185577393, 0.2405223399400711, 0.12647110223770142, 0.060619499534368515, -0.757036030292511, 0.38687559962272644, 1.3194066286087036, 0.3404563367366791, -0.5278943777084351, -0.4462894797325134, 0.8047292232513428, -0.27032798528671265, -0.29147183895111084, 0.40856683254241943, -2.0733044147491455, 0.3530914783477783, -1.3479501008987427, 0.05942763760685921, -1.7281666994094849, -0.3263447880744934, -1.1754910945892334, 0.762708306312561, 0.3088092505931854, -1.980338454246521, -1.3300001621246338, -1.3347368240356445, -0.738813579082489, 0.10745156556367874, -0.7077577114105225, 1.1997666358947754, 0.5151110887527466, -0.06544755399227142, 1.5724140405654907, 0.2620975971221924, -0.9620513319969177, -0.17097143828868866, -0.648520290851593, -1.0126845836639404, -0.4971168339252472, -0.9093418717384338, -0.8101299405097961, -1.7885342836380005, 0.5787346363067627, 0.34658756852149963, -0.47968927025794983, 0.26269909739494324, 2.2321410179138184, -1.543862223625183, 0.1855279505252838, 1.1072189807891846, -0.3905784785747528, 0.2927473783493042, 1.5408486127853394, -0.028791436925530434, 1.4561251401901245, -1.3992472887039185, -0.10856398195028305, 0.6373515725135803, -0.6662943959236145, 0.862191915512085, 1.424981951713562, 0.5866304039955139, 1.0779350996017456, -0.7477665543556213, 0.433651328086853, -1.867462396621704, 0.44394204020500183, -1.39552640914917, -0.632823646068573, -0.8691200017929077, -0.8322960734367371, 3.05403470993042, 1.7516655921936035, 2.2913870811462402, -0.5485286116600037, 1.2056479454040527, -1.1905829906463623, -0.6387106776237488, 0.9755430817604065, 0.4233904480934143, -0.3404602110385895, -0.2542601525783539, -0.48053625226020813, 0.0689380019903183, -1.4738861322402954, -1.683478593826294, -0.3565824031829834, 0.9583975672721863, -0.07452398538589478, -0.03697175905108452, -1.1962177753448486, -0.5032025575637817, -0.17691504955291748, -0.1689644455909729, 1.355761170387268, -1.6147223711013794, 0.6916895508766174, -0.7432196140289307, 1.7447067499160767, 1.2960166931152344, -0.858918309211731, 0.6044514179229736, 0.8730891346931458, 0.7787697911262512, 0.1100822165608406, 0.6222597360610962, -0.8254019021987915, 1.6398231983184814, -0.5054726600646973, -0.2220027595758438, 0.38313695788383484, 0.9185616970062256, 0.4336521327495575, 0.4122239649295807, -0.9670247435569763, -2.061548948287964, -0.6682587265968323, -0.5784914493560791, 1.7064282894134521, -0.6719930171966553, -1.0866631269454956, -0.9521028399467468, -0.5138468742370605, -0.9209223389625549, 1.383565902709961, -0.6146597862243652, -1.2537959814071655, -0.2693738043308258, -1.0314832925796509, -1.1688406467437744, 0.7551933526992798, 0.32451653480529785, -1.16339111328125, -0.4075348973274231, -0.5428004860877991, 0.3065233826637268, -0.8553792834281921, 0.3608875274658203, -0.8142989873886108, -0.12502193450927734, -0.7956984043121338, 0.06536637991666794, -1.3643182516098022, -0.6369243264198303, -0.07857222110033035, 0.04650541767477989, 0.6733303666114807, -0.03975291550159454, 1.0411946773529053, 0.74833083152771, 0.8287431001663208, 0.1765121966600418, -0.6074616312980652, -1.296215295791626, 0.5544224381446838, -0.7009519338607788, 0.8795321583747864, -0.8902800679206848, -0.720981240272522, 0.020331373438239098, 0.7417382597923279, 1.44212806224823, -0.9716205596923828, -0.539516031742096, 0.5671759247779846, 0.5072950124740601, 0.8432636857032776, 1.749863624572754, -0.3965461552143097, 0.18007004261016846, -2.215590000152588, -0.27794620394706726, -0.7104837894439697, 0.26309627294540405, -0.15119338035583496, 0.10271789878606796, 0.19929847121238708, -1.5584936141967773, 1.2379703521728516, 0.37722352147102356, 0.5302830934524536, -0.9348602294921875, 0.9871058464050293, 0.058006949722766876, 1.0692812204360962, -1.7407808303833008, 0.6742173433303833, -0.8196887373924255, 1.234237790107727, -0.0907682329416275, -0.11459225416183472, -1.9928456544876099, -0.4505474269390106, 0.0989241898059845, 0.4079503118991852, -1.8083374500274658, 1.27540922164917, 0.9300397634506226, 0.7354504466056824, 0.7519035339355469, -1.018052101135254, -2.066711187362671, -0.7515107989311218, 0.3978532552719116, -1.6060218811035156, -0.1404755711555481, 1.2793365716934204, 1.4252647161483765, 1.0947054624557495, 1.324747920036316, -2.2484264373779297, -0.28770890831947327, 0.4914638102054596, -0.3148916959762573, 0.6826126575469971, 0.6366207599639893, -0.07111574709415436, 1.7493062019348145, -1.1268222332000732, 2.9633214473724365, 0.3456820845603943, -0.5569579005241394, 2.046877861022949, -0.38296377658843994, -1.9428998231887817, -1.6741013526916504, -0.31058844923973083, -0.3497408926486969, -0.10212615132331848, 0.39354607462882996, 1.32162344455719, -0.9082795977592468, 0.03651896119117737, -1.3897829055786133, -0.293108195066452, -0.10278253257274628, 0.6836610436439514, -0.851669430732727, -0.7202638387680054, -0.6480444669723511, 1.204742431640625, -0.3952447175979614, 0.9497545957565308, 1.131842017173767, -0.27733463048934937, 0.39683133363723755, 1.1258167028427124, 0.8118810057640076, -0.8901508450508118, 1.1868191957473755, -0.5036868453025818, 0.45859140157699585, 1.3889329433441162, -0.6187259554862976, 1.540795922279358, 0.2797723412513733, -1.2208499908447266, 1.5284620523452759, 0.11109453439712524, -0.9158782362937927, 0.6087892055511475, -0.5561344027519226, 1.3788520097732544, 1.057440996170044, -0.2641483545303345, 1.4546087980270386, -1.14115571975708, -2.0171620845794678, 0.3816981613636017, -0.25310954451560974, 0.03741694614291191, -0.35793355107307434, 0.23389331996440887, 0.523483157157898, -0.6581811308860779, -1.0684963464736938, 0.7343419194221497, 1.2134021520614624, 0.9550273418426514, -0.6689936518669128, -0.6799277663230896, -0.5043796300888062, -3.359506130218506, -1.6452422142028809, -0.621049702167511, -1.2109832763671875, -1.0689337253570557, 2.5707733631134033, 1.0228261947631836, 1.1455185413360596, 0.9869463443756104, -0.6663620471954346, -0.6575762629508972, -2.030712842941284, -0.902505099773407, -0.5392276644706726, 0.19532202184200287, 0.1863633692264557, 0.27578282356262207, -0.7625827193260193, 0.38174566626548767, -2.0214293003082275, -0.8957946300506592, 1.408940315246582, 0.030549582093954086, -1.234490990638733, -0.2740143835544586, -0.018095126375555992, -0.7486122846603394, 0.15115311741828918, 1.4975464344024658, -0.2099132090806961, 0.2633707821369171, 1.0625855922698975, -0.8863299489021301, -0.24699538946151733, -0.7974137663841248, -1.1734352111816406, -0.3573307394981384, -1.1081199645996094, 0.860724151134491, -0.23098959028720856, 0.16606533527374268, -0.9667128324508667, 1.2919905185699463, 0.8845264911651611, 1.9890073537826538, 0.5163403153419495, 0.1685628592967987, -0.5417562127113342, -0.43883171677589417, 0.649185061454773, 0.6268603205680847, 1.1483081579208374, 0.6238759756088257, -1.0923395156860352, -0.44053298234939575, -0.656165361404419, 0.29447290301322937, -2.86161208152771, -1.556728720664978, 0.8251647353172302, -0.47140106558799744, 0.3073734641075134, -0.04218882694840431, 0.002759383525699377, -0.5652717351913452, -0.17070285975933075, -1.7808630466461182, 0.14486683905124664, 1.274841547012329, 0.5571520328521729, -0.5112141370773315, 0.167890727519989, 1.0519976615905762, 0.8774836659431458, 0.023852741345763206, 1.8001720905303955, -0.6577566862106323, -2.8331384658813477, 0.4656507670879364, 0.6402389407157898, 0.6118008494377136, -0.11782796680927277, -1.222029685974121, -0.5626710057258606, -1.626696228981018, -0.9363456964492798, 0.12786366045475006, -1.9569482803344727, 0.713504433631897, -0.240957573056221, -0.24679680168628693, -0.3951818645000458, -0.2131352722644806, -1.6602989435195923, -0.7517774701118469, -0.5527094006538391, -0.4935891926288605, -0.6268355846405029, 0.6899613738059998, 1.4279805421829224, 0.8607486486434937, -0.21667072176933289, -0.7034751772880554, 0.4314650595188141, 0.350879043340683, 1.6736809015274048, 0.3675602972507477, -1.0624608993530273, -0.7268508672714233, 0.14267529547214508, 0.3469747006893158, -0.6487687826156616, 0.34946590662002563, -0.2742167115211487, -1.554082989692688, -0.6445039510726929, -0.6174819469451904, -0.319921612739563, 0.4772372245788574, 0.31605714559555054, -0.3505471646785736, 0.7343686819076538, 0.47087928652763367, -0.7122026681900024, -0.5949966311454773, -0.7959951162338257, -0.36949819326400757, 0.4495413303375244, 0.4456733763217926, -0.9877630472183228, -3.577258586883545, 0.18264876306056976, 0.26314857602119446, -0.11374234408140182, 1.8095777034759521, 0.3746436834335327, -1.0880286693572998, -0.5271167755126953, -0.6757859587669373, 0.23128998279571533, -1.0919932126998901, -1.408504605293274, 0.3056069016456604, 0.056176528334617615, -1.5617557764053345, -0.18675747513771057, 0.23337708413600922, -0.7192859053611755, 0.762561559677124, -2.102198839187622, 0.36004263162612915, 0.6060048937797546, -1.3889280557632446, -1.6073085069656372, -0.5500087141990662, -0.2197018265724182, 0.5654080510139465, -0.9733218550682068, -0.061113059520721436, -1.0187957286834717, -0.14685216546058655, -0.11399294435977936, -0.1516123265028, 0.5846265554428101, -0.07722046226263046, 1.2343913316726685, -0.05255069583654404, 0.9711684584617615, -1.4737448692321777, 0.21927881240844727, 1.302488923072815, 1.0358378887176514, 0.40720316767692566, -1.9768754243850708, 0.1750372052192688, -1.0325703620910645, -0.3921666443347931, 0.9838155508041382, 0.4420899748802185, 0.4746226668357849, 0.4729454219341278, -0.594960629940033, 0.6950550079345703, 0.0038150534965097904, -0.8062216639518738, 1.1638832092285156, 0.7839351892471313, 0.741524875164032, -0.13000540435314178, 0.8685117959976196, 0.05185842886567116, 0.9818299412727356, -0.7324376702308655, -0.4150165617465973, 0.08811800181865692, 1.591826319694519, -1.413488745689392, 0.15590247511863708, -0.42231541872024536, -2.1309635639190674, -0.8822475075721741, 1.1362582445144653, -0.29670965671539307, 1.0169003009796143, -1.1537595987319946, 0.8754474520683289, 0.1799187958240509, -1.2782700061798096, 1.2015995979309082, 0.024970078840851784, 0.22152426838874817, 0.7200270295143127, -0.07987163215875626], [-2.0591397285461426, -1.489486575126648, 0.9366188049316406, -0.13887660205364227, 1.6360746622085571, -0.25576892495155334, -0.09475002437829971, -0.23586563766002655, -0.9853585958480835, 0.8705304861068726, -1.3882455825805664, 0.39619728922843933, 0.550560712814331, 0.28668782114982605, -0.09233078360557556, 2.013303279876709, -1.5866841077804565, 0.7261962890625, -2.4327120780944824, 0.45041364431381226, 0.10862144827842712, 1.09377121925354, 1.255091905593872, -0.7934756875038147, -1.1111980676651, 0.3663041889667511, 0.32253414392471313, -0.623680830001831, -0.925740122795105, 2.942605495452881, -0.7920504212379456, -1.235322117805481, -0.5474507212638855, 1.916994571685791, -1.252842664718628, 0.351322203874588, 0.8025065660476685, 1.3469784259796143, -1.0155242681503296, -0.2388256937265396, 0.23417450487613678, -3.5236148834228516, 0.9145674705505371, 0.27325624227523804, -0.188629612326622, -0.9266728162765503, -0.8773015737533569, 0.0715312734246254, 0.34568509459495544, -0.14694954454898834, 0.041655343025922775, 0.9267983436584473, -0.7874288558959961, 0.45862260460853577, 0.7805863618850708, -0.8229789137840271, -0.13898488879203796, -0.9274443984031677, -1.4954203367233276, 0.9875943064689636, -0.18752020597457886, 1.5208083391189575, 0.1911129206418991, 0.2678958773612976, -0.18154174089431763, 0.3022400438785553, 0.44771987199783325, 1.2205886840820312, 0.9611878991127014, 0.4156501889228821, 0.12472249567508698, -0.7336666584014893, -0.9869064688682556, -0.6028967499732971, -0.33649393916130066, -0.3853287100791931, 0.9645944237709045, 2.2381179332733154, 0.7490925788879395, -1.1545307636260986, -0.30934664607048035, 0.9813929200172424, 0.6764091849327087, 0.3561033010482788, -1.8281311988830566, -0.7139461040496826, 1.464540719985962, 0.8796599507331848, 0.588666558265686, 0.9847081303596497, -1.6260144710540771, 0.576973557472229, -0.16691359877586365, 0.04025301709771156, 1.4299976825714111, -0.8323396444320679, -0.9004427790641785, -1.0306200981140137, 0.9667887091636658, -0.868450403213501, 1.0400502681732178, -0.48197510838508606, 0.9340865612030029, -1.1129205226898193, -0.7615219950675964, -1.3334476947784424, 0.23685021698474884, 0.037806395441293716, 0.31584176421165466, -0.05667395889759064, -1.9263396263122559, -0.14581197500228882, -1.0440200567245483, 0.6847714185714722, 1.498992681503296, 1.1682498455047607, -0.6701700687408447, 0.10455616563558578, 0.15401238203048706, 0.22019274532794952, 0.6624765396118164, 2.8461875915527344, 0.5181906223297119, 0.3922156095504761, 0.6364123821258545, -0.3954833447933197, -2.4860026836395264, -0.35919731855392456, 0.9123930931091309, -1.1549417972564697, 0.9710986614227295, -0.8709726333618164, 0.9342121481895447, 1.517188549041748, 1.583406686782837, 1.7534148693084717, -0.7669417858123779, 0.9384823441505432, -0.1312326341867447, 0.3887866735458374, -0.9268163442611694, 0.4270288050174713, -0.17082230746746063, 0.7695336937904358, 0.7256911396980286, 1.325761079788208, -1.9960641860961914, -0.35695943236351013, 0.08679405599832535, -1.0542694330215454, -0.6852994561195374, 0.507926344871521, -1.1491564512252808, -0.19682790338993073, 0.521133542060852, -0.1783890426158905, 0.4837135970592499, 0.25331735610961914, -0.9830513000488281, 0.5827513933181763, -0.24477922916412354, -0.0628894567489624, -0.7253849506378174, -0.04515935853123665, -1.0561121702194214, 1.1020070314407349, -0.04687565565109253, 0.4172286093235016, -0.8565964102745056, 1.329393744468689, 2.2720441818237305, 0.6975464820861816, -0.6942359209060669, -0.037716642022132874, 1.9709420204162598, -0.21506132185459137, -0.5961718559265137, 0.34528422355651855, -1.1911207437515259, 1.2932112216949463, -0.039379511028528214, -0.26390671730041504, 1.4866023063659668, 1.0478954315185547, -2.5317060947418213, -0.44470253586769104, 1.1710307598114014, 0.44500622153282166, 0.35796502232551575, -1.7095224857330322, 0.31943073868751526, 0.6546719074249268, 1.5577298402786255, -0.07899023592472076, -0.17626053094863892, -0.9221283793449402, 1.431639552116394, -1.58518648147583, 0.593674898147583, -1.0929820537567139, -0.15887987613677979, -0.84612637758255, 2.51922345161438, 0.5003042817115784, -0.5352940559387207, -0.19858422875404358, 1.2000876665115356, 1.4500468969345093, 1.740996241569519, -0.1786620020866394, -0.22891101241111755, -0.6692955493927002, 1.4600896835327148, 1.0457559823989868, 0.16041113436222076, 0.25028014183044434, -0.2822417616844177, 0.32612350583076477, -0.8830665349960327, 0.660592257976532, 1.579939365386963, 1.0430338382720947, 0.9828901886940002, 1.5580826997756958, 0.9742873907089233, 0.9326404929161072, -0.716382622718811, 1.6531283855438232, -1.4627331495285034, -0.18356457352638245, -1.2669334411621094, 0.8949794173240662, -0.5269843339920044, -0.3194684684276581, -0.019166743382811546, 0.030917387455701828, 0.3941815197467804, 0.23955288529396057, 0.21763934195041656, -0.6365294456481934, -0.8337904810905457, -0.6613842248916626, -0.24837009608745575, -1.3660941123962402, -1.6842212677001953, 0.41272756457328796, -0.19493958353996277, 1.0708203315734863, -0.4822297990322113, 0.9967571496963501, 0.9302924871444702, -0.1486274003982544, -0.37259364128112793, 3.4699833393096924, -0.5814527869224548, -2.662459373474121, -0.850193977355957, 0.015200478956103325, 0.3512561023235321, -0.6890386939048767, -0.2816006541252136, -2.0675806999206543, -1.6492114067077637, 1.0758733749389648, -0.49167320132255554, 0.5951018929481506, 0.35862991213798523, -0.22020862996578217, 0.2842349708080292, -2.047470808029175, 0.21808375418186188, 0.8477648496627808, 0.13748866319656372, -0.47587019205093384, 0.6456907987594604, 1.0066410303115845, 0.04802003502845764, 1.377238154411316, -0.25897687673568726, -0.7154250144958496, -0.14448322355747223, -0.9072619676589966, -0.01835237257182598, -0.351924866437912, 0.23325993120670319, 2.0625991821289062, 1.0003247261047363, -0.5079030990600586, -0.27391648292541504, -0.64042067527771, -0.3678780794143677, -0.07526303082704544, 1.1532846689224243, 1.0651588439941406, -0.02441728487610817, -1.0542806386947632, 0.5884740948677063, 1.038312315940857, -0.32386013865470886, -0.12444381415843964, 0.6130519509315491, -1.636281132698059, -0.8045589327812195, 2.4802002906799316, 2.7806460857391357, -1.4594225883483887, -0.9047861099243164, -0.0869249701499939, -0.988994300365448, -0.40954920649528503, 0.2680339515209198, -1.5350136756896973, -1.1862874031066895, -0.1666017770767212, 0.6386768221855164, 1.660934567451477, -1.431107759475708, -0.7263853549957275, -0.20054589211940765, -1.1719064712524414, -1.1824690103530884, 0.3589385449886322, 1.0000780820846558, -1.4839569330215454, 0.45098772644996643, 0.23172278702259064, 0.5270175337791443, -1.5973381996154785, 0.3348861336708069, 0.07281206548213959, -0.16150714457035065, -1.7744802236557007, 0.1862565279006958, -0.5843074321746826, 0.46528545022010803, -1.00580632686615, 0.31573182344436646, 0.21445991098880768, -0.4732145369052887, 0.3360578715801239, 1.305470585823059, 1.116499900817871, 0.2937479615211487, 1.2597129344940186, -1.316104531288147, 0.08978521078824997, 1.3497416973114014, 0.4180371165275574, 0.4838365614414215, -0.8974602818489075, -0.3432159125804901, 0.9020836353302002, 0.82475745677948, -0.0015972788678482175, -1.032645344734192, -0.24508686363697052, 0.579253077507019, -0.13205620646476746, -0.8190882205963135, -1.299782156944275, -0.1345328688621521, 1.4656866788864136, -0.42940250039100647, -0.5295336246490479, -0.420474112033844, 3.046069860458374, 0.3986571133136749, -0.635202944278717, -3.0021555423736572, -0.3129366934299469, 0.6023772954940796, -0.11402307450771332, -1.7035635709762573, -0.7230844497680664, -1.6875685453414917, 2.832381010055542, 0.2334933876991272, 0.7192900776863098, -1.8526431322097778, -0.1836346983909607, -0.6876755952835083, -0.78483647108078, 1.1530202627182007, -1.6274669170379639, -0.04381025582551956, 0.7170314788818359, -0.46710124611854553, 0.03491373732686043, 0.6383074522018433, -0.5702458024024963, 0.9978396892547607, -1.107701301574707, -1.4888910055160522, 1.1196562051773071, 0.9994416236877441, -0.6900885701179504, 0.5625003576278687, 1.4061146974563599, -1.0124231576919556, -0.2615949213504791, -1.0628315210342407, 2.300163745880127, -0.8340728282928467, -0.16044460237026215, 0.5201441645622253, 0.13729903101921082, 1.1407520771026611, 0.02782224863767624, 2.7049896717071533, -1.1356531381607056, -0.44327524304389954, -0.08619674295186996, 0.4304794669151306, -0.3600103259086609, 0.24921754002571106, 0.38896438479423523, 0.6858629584312439, -1.7076541185379028, 0.8628563284873962, 0.5235715508460999, -0.38184356689453125, 0.9801624417304993, -0.41068148612976074, 1.1093045473098755, -0.19626367092132568, 0.15768222510814667, -0.45743876695632935, 2.5144600868225098, -0.2407345324754715, 0.6697661876678467, -0.4301266670227051, -1.2352373600006104, 1.195120096206665, -0.43850332498550415, -1.6066826581954956, -1.1740106344223022, 1.3104780912399292, -1.6155937910079956, 1.24631929397583, -1.3722023963928223, -1.093133807182312, -0.5264802575111389, 0.4997349977493286, 0.3117985725402832, -0.2051272690296173, 1.3039491176605225, -1.0082119703292847, 0.07054361701011658, -0.32042059302330017, 0.4036456346511841, 1.837860107421875, 2.2290806770324707, 0.9718974828720093, 0.3553750216960907, -1.0748751163482666, -0.623683512210846, -0.5302674770355225, 0.4902910590171814, -1.0139498710632324, -1.248550295829773, 0.7504494190216064, -2.0123848915100098, 0.08006748557090759, -1.3220641613006592, 0.9946256279945374, 1.8457914590835571, 0.8561778664588928, -0.5249332189559937, -0.47147196531295776, -0.12090109288692474, 0.18604739010334015, -1.6862716674804688, -0.7288249135017395, 0.2383616715669632, -0.6799944639205933, -0.38765594363212585, 0.5719772577285767, 1.1595319509506226, -1.5901952981948853, 0.03422857075929642, 0.025158995762467384, -0.016875185072422028, 1.1122567653656006, -1.6378403902053833, 0.016875388100743294, 0.11707860976457596, -2.0377919673919678, 0.9616256952285767, -1.290871024131775, 0.8582239151000977, -0.5887638926506042, 0.5833815336227417, -0.07247628271579742, -0.837240993976593, 0.21925796568393707, 1.7139474153518677, 2.4014668464660645, 0.1749965101480484, -1.0557199716567993, 1.6943016052246094, -0.8208074569702148, -0.503108024597168, 0.6612508893013, -1.1211248636245728, -1.1797479391098022, 0.9774168133735657, -1.1641418933868408, 0.0487063005566597, -1.4754927158355713, 0.1270843893289566, 1.2503910064697266, -1.132767677307129, -0.410331666469574, 0.21922683715820312, -1.747929334640503, -0.5189192295074463, 0.2450403869152069, -0.503604531288147, -1.3004546165466309, -0.09995609521865845, -1.0187301635742188, 1.4488465785980225, 1.1450059413909912, -1.3174458742141724, 1.5232038497924805, 0.2442888766527176, 0.8741253614425659, 0.25291818380355835, -0.6753157377243042, -1.485421895980835, -0.6864676475524902, -1.4073694944381714, 1.642735481262207, 1.2264481782913208, 0.8653647899627686, -0.5805113911628723, -1.2453535795211792, 0.5160129070281982, 0.6019882559776306, 0.313271164894104, -1.23935866355896, 0.5334004759788513, -1.2799149751663208, 0.18231676518917084, 0.8361153602600098, -0.21100379526615143, -0.12873363494873047, -0.9554147720336914, 0.5272420048713684, 0.046327825635671616, 0.8510305285453796, -0.22455327212810516, 0.2385524958372116, -0.3626646101474762, -0.6704230904579163, 1.8813445568084717, 0.3048896789550781, 0.12446839362382889, -1.2366682291030884, -1.6581511497497559, 1.344892978668213, -0.5994998812675476, -0.04511595144867897, 0.9769418239593506, -1.0945886373519897, -0.6904665231704712, -0.6651883721351624, -0.5830745697021484, -1.8903436660766602, 0.8165162801742554, -0.9109324216842651, -0.27260345220565796, 0.38491764664649963, -1.3386386632919312, 0.037988435477018356, -0.43215349316596985, -0.9824060797691345, -1.0316053628921509, -0.1596294641494751, 0.9182664155960083, -0.06363753229379654, 0.10970434546470642, -2.2404701709747314, -1.0090099573135376, -2.399153470993042, -1.3498457670211792, 2.3358027935028076, 0.12875793874263763, 0.19725556671619415, -1.5341991186141968, -0.3071834146976471, -0.6559625864028931, -0.5222122073173523, 0.4253441095352173, -0.3468090295791626, 0.7931225299835205, 2.2444679737091064, 0.12111296504735947, 0.15019629895687103, -0.1833794116973877, -1.6797183752059937, 0.8118237853050232, -0.13343669474124908, -1.0853984355926514, 0.06060418114066124, 1.2504619359970093, -0.3807656764984131, -0.11246137320995331, -1.2304091453552246, -0.01098922360688448, 0.07849016040563583, 0.34133556485176086, 1.371094822883606, -0.5548161268234253, -0.1848369836807251, -0.42461302876472473, -0.6182721853256226, -0.46947214007377625, 0.398980051279068, -0.13208279013633728, -0.40766164660453796, -0.6157594323158264, 0.7496605515480042, 1.5561296939849854, -0.5212216377258301, 1.4827284812927246, 0.49412351846694946, -0.8469630479812622, -0.28667330741882324, 1.029802918434143, -1.5843470096588135, 0.1151905283331871, -0.05653410032391548, 0.3196888566017151, 0.5554990768432617, 0.178230419754982, -0.018077917397022247, -0.6747651696205139, -1.0413116216659546, -0.5130003094673157, 0.04945552349090576, 1.0472040176391602, -1.4098535776138306, 0.6712620258331299, -1.5527399778366089, -0.6179649233818054, 0.29527413845062256, 4.069230079650879, 0.3073432445526123, 0.7510138750076294, -1.2037358283996582, 0.9888923168182373, 2.049873113632202, 0.7660781145095825, -0.8656858801841736, -0.6036157608032227, 0.472713440656662, 3.322869062423706, -0.09936069697141647, -0.5407734513282776, -0.5803810358047485, 0.5615197420120239, -2.331366539001465, 0.8838416934013367, -0.40783557295799255, 0.2555505037307739, 0.21479769051074982, -0.662635862827301, 0.6313237547874451, -1.1630687713623047, 1.2440712451934814, -0.27889665961265564, -0.7685768604278564, 0.7485619783401489, -0.16847674548625946, -1.149768590927124, 1.5896763801574707, -1.9891537427902222, -0.2943725883960724, 1.8757888078689575, -1.1215780973434448, -0.6545491218566895, 0.2376907467842102, -1.0869899988174438, 0.7107691764831543, -0.1270459145307541, 0.7415919899940491, 0.6260558366775513, 0.8522188663482666, 0.16821616888046265, -0.16663967072963715, 0.30621856451034546, -0.892886221408844, 0.08212828636169434, -1.1383041143417358, -0.18353888392448425, 1.2417799234390259, 0.055735714733600616, -0.9627057313919067, 0.23260466754436493, -1.466733455657959, 0.8942757248878479, 1.0490005016326904, -0.4230329990386963, 0.09501225501298904, -0.9053833484649658, 0.7956568002700806, 2.7705140113830566, -0.32290586829185486, -2.1018128395080566, -0.8538199067115784, 0.45607343316078186, -0.16579248011112213, -1.467095136642456, -1.5305180549621582, 0.31242695450782776, 0.6320472359657288, 0.80025315284729, -1.1797722578048706, 1.0692194700241089, -0.7165540456771851, 0.5997322797775269, 0.668897271156311, 0.020764125511050224, -0.9116621017456055, -0.6179057955741882, 0.6681103110313416, 0.9773414731025696, 0.7093377709388733, -0.25578609108924866, -0.9038727283477783, -2.3446149826049805, -0.8777952790260315, -0.29782193899154663, 0.4836863577365875, 1.9487292766571045, -1.629132628440857, 0.10103842616081238, 0.3924555778503418, -0.1272427886724472, -0.1984775811433792, 1.0041427612304688, 0.38613900542259216, 1.153342843055725, 0.3200823962688446, -0.43563398718833923, -1.9427067041397095, -2.7744979858398438, 0.7245782613754272, -0.11286728084087372, 1.2580337524414062, 0.020337358117103577, 1.299149990081787, 1.4970980882644653, -0.594103991985321, -0.20109884440898895, 0.23719824850559235, 1.015148401260376, -1.4764069318771362, -0.9737378358840942, 1.6003532409667969, 0.3327014744281769, 0.358060359954834, 0.026397548615932465, -0.11198129504919052, 0.925179660320282, 1.0661839246749878, -0.1705373227596283, -0.5492266416549683, -0.8827592134475708, -1.9464808702468872, 1.1286876201629639, -0.4012589752674103, -0.9301570057868958, 0.24308712780475616, 0.717273473739624, -1.704271674156189, -1.2963347434997559, -0.5734215378761292, -0.45311930775642395, 0.6750818490982056, -1.8567986488342285, 0.7750756740570068, -1.3286731243133545, 0.6377379894256592, -1.4720134735107422, 0.3945459723472595, 0.603425920009613, -0.08717627823352814, -0.003833634778857231, 1.1374350786209106, 0.21497730910778046, 2.7266979217529297, -0.21582114696502686, 1.477269172668457, -0.42893651127815247, -0.40914055705070496, 0.8419923782348633, 2.507081985473633, 0.8553490042686462, -0.7073794007301331, -1.6094083786010742, 0.6450011730194092, -2.547506809234619, 1.0474973917007446, -0.10376381129026413, 1.4670920372009277, 0.5842161178588867, -1.0390238761901855, -0.7296632528305054, -0.8773664236068726, -1.2595999240875244, -1.6628453731536865, 1.190428376197815, 1.0580137968063354, -1.4147175550460815, -2.203878879547119, -0.2528526186943054, -1.1040399074554443, -0.3039354979991913, 1.917297601699829, 0.8977351784706116, -0.8362410068511963, -0.19874995946884155, -1.0114059448242188, 0.5027334690093994, 1.1746824979782104, 2.4507107734680176, -0.17860926687717438, 0.5822981595993042, -1.3606069087982178, 0.9525308609008789, -1.3157238960266113, 1.777512788772583, 0.8279579281806946, 0.6429236531257629, 1.0828406810760498, 0.15479445457458496, 0.4393762946128845, -0.4375055432319641, 0.33436790108680725, -0.42329469323158264, 0.6268353462219238, 0.7542901635169983, 0.0515863299369812, -0.7081902623176575, 0.0319223552942276, -0.7807615995407104, -0.3592897355556488, 0.7764521241188049, 0.1354028284549713, -0.41788217425346375, 0.7838311791419983, 0.4131796061992645, -0.6113467812538147, -0.6459271311759949, 0.06357179582118988, -1.9391181468963623, 1.0181337594985962, -0.07228344678878784, -0.08581072837114334, 0.514885663986206, -1.6258385181427002, -0.8822515606880188, 0.02482100948691368, -1.117692470550537, -0.530757486820221, 0.29883328080177307, 1.8336477279663086, -0.8595075607299805, 0.02915186807513237, 1.1656947135925293, -0.8253276348114014, 0.6465716361999512, -1.1949063539505005, 0.9400056600570679, 0.7372671365737915, 2.090904474258423, -1.6495112180709839, -1.2942967414855957, -1.4691097736358643, -1.2685765027999878, 0.6878462433815002, -0.6040093898773193, -0.04894738644361496, 1.1160836219787598, -0.9428884983062744, 0.5380375385284424, -0.43188750743865967, 1.2183088064193726, -0.09421156346797943, 0.20786626636981964, 1.9186944961547852, 1.7170735597610474, -0.6065490245819092, -0.2146129459142685, 1.0017884969711304, 0.8638944625854492, -1.1774574518203735, -1.2096550464630127, -0.8643699884414673, -0.20373739302158356, -1.320145845413208, 0.578350841999054, -0.8033785820007324, 0.3017510771751404, 2.32869029045105, -1.2761924266815186, 0.5119799375534058, -1.1282751560211182, -0.9682638049125671, -1.0046045780181885, 0.1448563188314438, 0.9235988259315491, 0.4084075689315796, -1.215570092201233, -0.7865965962409973, -1.088873267173767, -0.06858653575181961, -0.7614924907684326, 1.4324012994766235, 0.4555507302284241, -0.8998508453369141, -2.250720977783203, 1.4436347484588623, -1.3228306770324707, 0.8239061236381531, 0.3715443015098572, -0.3520109951496124, 0.0006398811819963157, 0.9827725291252136, 0.5445737838745117, -0.010245050303637981, -0.5418543219566345, 0.129924938082695, -0.04423677176237106, 0.6986716985702515, -0.5070164799690247, -0.9285247921943665, -2.1184213161468506, 1.2797907590866089, 1.0834654569625854, 0.010480543598532677, -1.173224687576294, -0.896803081035614, 0.3336571753025055, -0.9309936165809631, -0.7057557106018066, -1.3709670305252075, -0.8568210601806641, -0.8338523507118225, 1.315814733505249, 0.0291141364723444, -1.968029260635376, 0.5178624987602234, -0.4721737205982208, 0.04111483320593834, 1.7748152017593384, -0.2778875529766083, -1.1035752296447754, -1.4392915964126587, -0.8814265131950378, 0.07490754872560501, -0.4904387891292572, 1.960669994354248, 0.7570390701293945, -0.37529104948043823, 0.31981849670410156, 0.8321154713630676, 0.22976365685462952, 0.503989040851593, -0.11659775674343109, -0.3664788603782654, 0.648105800151825, 1.8134608268737793, -0.3521674871444702, -0.44351130723953247, -1.5741617679595947, 0.35780441761016846, -0.7611401081085205, -1.2310786247253418, -2.392749309539795, -1.3791160583496094, -0.7751284241676331, -1.1021082401275635, 0.4681209325790405, 0.27670228481292725, 0.627911388874054, 1.0166220664978027, -1.3383907079696655, 0.40963029861450195, -1.522403359413147, -0.12832048535346985, 0.29676324129104614, -1.084561824798584, 0.2867843508720398, 1.1156580448150635, -0.2247614860534668, -0.06848884373903275, 0.032304342836141586, 0.2467721402645111, -0.7514186501502991, -0.4433229863643646, 0.40035486221313477, 1.2791768312454224, -2.079237937927246, -0.12572269141674042, 0.980108916759491, -0.38827624917030334, 2.050227403640747, -0.13768738508224487, -2.050901412963867, 0.8365675210952759, -0.6839577555656433, 0.10241478681564331, -0.21851065754890442, 0.1708083599805832, -1.4696154594421387, 0.03371964395046234, -0.5324503779411316, -0.4023169279098511, -1.126284122467041, -0.3939174711704254, 0.38737547397613525], [-0.512971818447113, -0.42459720373153687, -0.4173365533351898, 2.1630778312683105, 1.3331624269485474, 0.9813542366027832, 1.1168497800827026, -1.3024046421051025, -2.8066554069519043, -1.3452014923095703, -2.4100937843322754, 1.8550853729248047, 0.46912169456481934, 0.8100667595863342, -1.4042556285858154, 0.6437721848487854, 0.11598305404186249, 0.851198136806488, -1.2560442686080933, -0.7037091851234436, -0.1754038780927658, -0.9072219729423523, -0.9818904399871826, 1.0151557922363281, -0.6147201061248779, 1.0144000053405762, -1.585669755935669, -0.6497898697853088, -0.169866144657135, 1.290170431137085, -1.4872719049453735, 0.09529532492160797, -0.2698817849159241, 0.24278470873832703, -1.1329971551895142, -1.4057912826538086, -0.4777510464191437, 0.025876445695757866, 0.04973580315709114, -0.10694784671068192, -3.240497350692749, -0.8459905385971069, -0.7956538796424866, -0.6669684052467346, -0.39689430594444275, -0.014934012666344643, 0.820709764957428, 1.3588629961013794, -2.5105113983154297, 1.9296756982803345, 0.14169178903102875, -0.44626912474632263, 1.1381124258041382, -0.6191702485084534, 1.0546939373016357, -1.4986820220947266, -1.2306143045425415, -0.2735539376735687, -1.3763370513916016, 0.24855050444602966, -1.0539792776107788, -0.24109961092472076, -1.80440354347229, -0.38025230169296265, -2.4432811737060547, 0.1947699785232544, 0.7783641219139099, -0.06087362393736839, 0.7770804762840271, 0.5832937359809875, 0.22528794407844543, -1.1141375303268433, 0.362265020608902, -0.3802317678928375, 0.11128146201372147, -1.19756019115448, 0.33982306718826294, -1.7428427934646606, 1.8798762559890747, -0.22106608748435974, -0.1236906349658966, -0.45904824137687683, -0.2823649048805237, -0.10039958357810974, 0.07130547612905502, 0.3007729947566986, -0.5334749221801758, -0.2298988550901413, 1.6549781560897827, -0.6047990918159485, -0.45167773962020874, -0.6283904314041138, -1.0185803174972534, 1.421533465385437, 0.444363534450531, 0.03733125701546669, 0.5918064713478088, -0.36338508129119873, -1.26291024684906, -1.3241147994995117, 0.23325829207897186, -0.5711646676063538, -0.5972866415977478, -1.0739545822143555, -1.100498914718628, -0.18289239704608917, -1.615248441696167, -0.19792287051677704, -1.3041132688522339, 0.17445924878120422, 1.9883300065994263, -0.9927999377250671, 0.47314679622650146, 0.5030149221420288, 1.3464175462722778, 1.962585687637329, 0.949006974697113, 1.009160041809082, -0.7927309274673462, -1.1374982595443726, 0.7273396253585815, -0.12698696553707123, -1.5216453075408936, -1.7203114032745361, 0.8638809323310852, -1.8062244653701782, -0.7755992412567139, 1.3299124240875244, -0.5790014863014221, 1.5408909320831299, -0.5916661620140076, 0.010041557252407074, -0.968774676322937, 0.22382159531116486, 1.2726918458938599, 1.2449842691421509, 0.9027840495109558, -0.3511921465396881, 0.25679343938827515, -0.7908366322517395, 0.9146524667739868, -0.4101276099681854, 1.5341193675994873, -0.15174630284309387, 0.6015515923500061, 0.4286655783653259, -0.9262793064117432, -0.07427637279033661, -1.0517687797546387, -0.16559436917304993, 1.0233113765716553, -1.8829972743988037, -1.9843727350234985, 0.31888070702552795, -0.14553584158420563, -0.868657648563385, 0.199676051735878, 0.7096843123435974, 1.015097975730896, -1.421201467514038, -0.06340720504522324, 0.049686480313539505, -1.6116173267364502, 0.5938430428504944, -0.8312525749206543, -0.8255364894866943, -1.5122520923614502, 1.1315299272537231, -0.7663524150848389, -1.8080497980117798, 0.4949696958065033, -0.7171748280525208, -2.1567277908325195, -1.5090134143829346, 1.686833381652832, 0.43393760919570923, -0.9379661083221436, -0.812732994556427, 0.869886040687561, 0.2477540522813797, 0.25419023633003235, -0.899519145488739, -1.8503459692001343, 0.15126356482505798, 0.5199145078659058, -0.3125411868095398, 1.6740925312042236, 0.1712922304868698, 1.3427618741989136, -0.27096930146217346, -0.13360653817653656, 0.8212689161300659, 1.6036887168884277, -1.2433058023452759, 0.42394423484802246, 0.018071426078677177, -0.5708472728729248, 0.4004589915275574, -0.6458448767662048, -0.4621414542198181, 0.5335493087768555, 0.8295249938964844, 1.0044984817504883, 0.6004487872123718, -0.3931938111782074, -0.6275094747543335, -0.8316604495048523, -0.37459123134613037, -0.8888170123100281, -0.7574073076248169, -1.095815896987915, -0.09956210106611252, 0.5138849020004272, -0.5058680176734924, 0.23396740853786469, -0.439255028963089, -0.11805887520313263, -0.5264305472373962, 0.8107184171676636, 1.3921083211898804, -0.027567418292164803, 0.7153683304786682, 0.859454870223999, 1.0491085052490234, 0.2093603014945984, 0.3848382532596588, 2.0956952571868896, 0.2256999909877777, -0.7924037575721741, 1.148923397064209, 0.4735499620437622, -0.7191959023475647, -0.048683080822229385, 1.083876609802246, -1.8716872930526733, -0.401323527097702, -0.34486812353134155, -0.6188876032829285, -0.3563552498817444, -0.2723807096481323, -0.47342804074287415, 1.1658356189727783, -0.7023884057998657, 0.6712850332260132, 0.15907685458660126, 0.07144538313150406, -0.758002758026123, -0.6308627724647522, 0.07675927877426147, -0.129920095205307, -0.48284658789634705, 0.27038389444351196, -0.5522467494010925, 0.3628380000591278, -0.24686270952224731, 0.8046503663063049, -0.2569650411605835, -1.772884726524353, 2.1940627098083496, -0.7079234719276428, -2.4384214878082275, 2.3889269828796387, -0.4604399800300598, -1.524093747138977, -0.3206285536289215, 0.023110268637537956, 0.6573853492736816, -0.8744386434555054, 1.291146993637085, 2.9685239791870117, 2.0363383293151855, -0.2492748647928238, 0.6824684739112854, 0.19536368548870087, 0.525218665599823, -1.2145435810089111, 1.9333076477050781, 1.2378969192504883, -0.07482562959194183, -0.7704058885574341, 1.3703869581222534, -0.0004267469921614975, -1.4367413520812988, 2.4582319259643555, -1.0122103691101074, -0.13685137033462524, 0.2714482247829437, 0.24971400201320648, -0.761029839515686, 0.5820692181587219, -0.6258687376976013, -0.18475782871246338, -1.97796630859375, 0.7770066857337952, 0.8793716430664062, -0.6816204786300659, -0.31685870885849, 0.6020260453224182, 0.8819839954376221, -0.04173655807971954, 0.892575740814209, 0.10624951124191284, 2.043283224105835, 0.07692258059978485, -1.3875705003738403, 1.4407343864440918, 0.26166316866874695, -0.011905379593372345, -0.07201718538999557, -1.1290286779403687, 0.5673676133155823, -0.38829305768013, 0.20365023612976074, -0.2253955453634262, -0.23583145439624786, -0.040227729827165604, -0.07130369544029236, 0.9857062101364136, -1.1612590551376343, -0.43101558089256287, 0.1784723401069641, 0.5721262097358704, 0.07634443044662476, 1.5587071180343628, -0.6185215711593628, -0.15977483987808228, 0.8550830483436584, -2.6784658432006836, 1.062698245048523, -1.0545289516448975, -0.3550015985965729, -0.2502909004688263, -0.6078037023544312, -0.22046856582164764, -0.2214403599500656, 0.9213846921920776, -0.18747079372406006, -0.13649751245975494, -0.03499973937869072, 0.521213948726654, -0.9351275563240051, -1.6507242918014526, 0.18263515830039978, -0.5445590019226074, -0.6488367319107056, 2.1205661296844482, -1.0234131813049316, 0.5808408856391907, 0.13987118005752563, 0.06839918345212936, -0.1707734763622284, 0.33188310265541077, -2.247199058532715, -0.49048230051994324, -2.2338058948516846, -1.4188697338104248, -0.0588613897562027, 2.5715718269348145, 0.9220995306968689, 1.5263068675994873, -0.5755206942558289, 1.4731227159500122, 0.19515985250473022, -0.2138518989086151, -0.8913513422012329, -0.15152433514595032, -1.7866630554199219, -0.24364560842514038, 3.067695379257202, -1.2699795961380005, 0.23968909680843353, -0.8232098817825317, 0.8211326599121094, 0.6682155728340149, 0.7673996686935425, -0.6750273108482361, 1.8668795824050903, 0.3438115417957306, 1.3318272829055786, 1.7910720109939575, 0.7703675627708435, 0.338533878326416, 1.7397130727767944, -1.4176745414733887, 0.5773276090621948, -0.06109460070729256, -0.42766425013542175, 0.349731981754303, 0.9843717813491821, -1.1720679998397827, 0.9832586050033569, -0.8211697936058044, 1.5694555044174194, 0.8396697044372559, 1.8670263290405273, -0.280636191368103, 1.1149590015411377, -0.8093999624252319, -0.5630019903182983, 0.5999215245246887, -0.642309844493866, 0.12695375084877014, 0.8376613259315491, -1.285355806350708, 0.017338993027806282, 1.5571842193603516, -0.5939457416534424, 2.1429243087768555, -2.1416516304016113, -1.1751776933670044, 0.48994067311286926, 2.2859690189361572, -0.8454738259315491, -0.47447288036346436, 1.2378095388412476, 2.1989638805389404, -0.2800767719745636, 0.560930609703064, 2.8816311359405518, -0.503223717212677, 1.0221266746520996, 1.1180306673049927, 7.652385829715058e-05, -0.15822669863700867, 1.0907933712005615, 1.5542651414871216, -0.038700561970472336, 0.5293014049530029, -0.29962897300720215, -1.6508958339691162, -0.4504850208759308, 1.1634647846221924, 0.9412340521812439, 1.6961864233016968, -0.19384698569774628, -0.6099786758422852, 0.1776483803987503, 0.6511554718017578, -0.8175946474075317, 0.4000941216945648, 0.351083368062973, 0.8055803179740906, 1.8874943256378174, 0.5002173781394958, -0.6790966987609863, -0.7464267015457153, -0.6210918426513672, -1.0691643953323364, 0.707217812538147, -0.040842205286026, -0.5513968467712402, 0.7024733424186707, -0.6622130274772644, 0.606287956237793, 0.9644456505775452, -1.0236189365386963, -1.6262329816818237, 1.3827153444290161, -1.094637393951416, -1.0802645683288574, 2.2941982746124268, -0.7086421847343445, 0.20673169195652008, -1.514295220375061, 0.6010903120040894, -0.32351386547088623, 0.6880273818969727, 1.0067660808563232, -0.7428460717201233, 0.25377732515335083, 0.5289631485939026, -1.733379602432251, 0.21079304814338684, 1.6277110576629639, -0.9543807506561279, 0.4356266260147095, -0.5423039197921753, 1.6854528188705444, -1.5820931196212769, 1.6365164518356323, -0.8251290917396545, 1.5486382246017456, -1.5550785064697266, 1.2195930480957031, -1.4610481262207031, -0.834144115447998, -0.7094512581825256, 0.5265488028526306, -0.16772828996181488, -0.705241322517395, 0.6865108013153076, 0.06863086670637131, 0.4499361217021942, 0.6547203063964844, -0.0067605082876980305, -0.7487238645553589, 0.7532693147659302, 0.5069383978843689, -0.9542689919471741, -1.5274441242218018, -0.1890440732240677, -0.9503205418586731, 0.8045101165771484, -1.1360176801681519, -0.9611029624938965, -0.34387636184692383, -0.06855225563049316, -1.2079362869262695, -0.31132587790489197, 0.5297362804412842, -0.033501360565423965, 0.25641465187072754, 1.0083272457122803, 0.38192349672317505, 0.26789864897727966, 0.0560762844979763, 1.3023065328598022, -0.3603222370147705, 0.3598621189594269, 0.8101829290390015, 1.926777958869934, 0.23889367282390594, 0.7458227276802063, -0.8216298818588257, -0.4839998781681061, 0.7675656676292419, 1.1431714296340942, 1.9588346481323242, -1.4186184406280518, 0.8787355422973633, 1.051527976989746, -1.0384382009506226, -1.280987024307251, 0.7623986005783081, -0.18357495963573456, 0.0027255055028945208, 2.377520799636841, 0.1255173236131668, 2.0058982372283936, -0.31286513805389404, 1.6456573009490967, -1.1166799068450928, 0.47569137811660767, 0.687358558177948, -2.3811042308807373, 1.0242058038711548, -1.4110041856765747, -1.0942373275756836, 0.21080321073532104, 0.01504844706505537, 0.5370386838912964, 0.12237720936536789, 0.7797027230262756, -1.5322377681732178, 0.6299718022346497, -0.5059663653373718, -0.3444069027900696, 0.28094905614852905, -0.228116974234581, 0.9377879500389099, -1.3776799440383911, -0.6263747811317444, 1.6745399236679077, 0.49620509147644043, -0.27680277824401855, 0.900631308555603, -0.007803787477314472, 1.1588757038116455, -0.31626826524734497, -0.8694742321968079, -1.4883692264556885, -1.5815362930297852, -0.6113548278808594, 1.0382996797561646, 0.06086753308773041, -0.2776215374469757, -0.37698259949684143, -0.6496537923812866, 1.0641640424728394, 0.6810255646705627, -3.006740093231201, 0.12716621160507202, -0.6919804811477661, 0.4871402978897095, -0.02955560013651848, 1.8863602876663208, 1.321164608001709, 0.894966185092926, 0.7105445861816406, 0.8344293236732483, 0.8580095767974854, 1.0885220766067505, -1.2148691415786743, 0.5342855453491211, -1.0547904968261719, -0.3738517761230469, -0.23982654511928558, -0.5881729125976562, -0.6417509913444519, -0.6453354954719543, 0.504301905632019, 0.07912036776542664, 0.25118958950042725, 0.7062724828720093, -0.15028545260429382, 0.5186190009117126, -0.31932395696640015, 1.0368897914886475, -0.5944401025772095, -0.2029803842306137, -1.561422348022461, -0.15246497094631195, 1.8974941968917847, -1.3319389820098877, 0.6615638732910156, -0.40254175662994385, 0.34591200947761536, 0.1529242992401123, 0.2810859978199005, -0.17577265202999115, 0.9062154293060303, -0.43449723720550537, -0.31408143043518066, -1.3823277950286865, 0.02604604884982109, 1.4017435312271118, 0.9575031995773315, 0.6828293204307556, 0.9624778628349304, 0.3897430896759033, -0.35814476013183594, -0.11710518598556519, 0.13386958837509155, -0.36311352252960205, -0.9565856456756592, 0.620627760887146, 1.3288609981536865, -0.24592924118041992, 1.7180863618850708, -1.2760096788406372, 1.2565665245056152, 2.7373316287994385, 0.5150271654129028, -0.7661007642745972, -0.5228573083877563, 1.0352308750152588, -0.28453534841537476, 2.105381727218628, 1.620417594909668, 0.28561505675315857, -1.5350252389907837, 0.5970485210418701, 0.3341991901397705, -1.0525596141815186, -0.6579146385192871, 0.010878034867346287, -0.8123327493667603, -0.038805700838565826, 2.123444080352783, 0.15130409598350525, 1.1832853555679321, 0.11819466203451157, 0.31991812586784363, -0.5127692222595215, 0.8972896933555603, 0.823922872543335, -0.7197138667106628, -0.47720664739608765, 0.0642414316534996, -0.660176157951355, 1.292640209197998, -0.9954149723052979, 0.4780483841896057, -0.41480013728141785, 0.8256316781044006, -1.0149452686309814, 1.8582812547683716, -0.4412823021411896, 1.9817143678665161, -0.4992403984069824, -1.134934902191162, 0.20277157425880432, 0.7585012316703796, -1.471718430519104, -0.30342984199523926, -1.2218626737594604, 1.275696873664856, -0.36535152792930603, 0.9625367522239685, 1.0339009761810303, -1.9289547204971313, 0.09685717523097992, 0.16976433992385864, 1.402144432067871, -1.7831697463989258, 0.3194597661495209, -1.0937492847442627, 0.11207503825426102, -0.23424051702022552, -0.9326702952384949, -1.4795466661453247, -0.551935076713562, 1.462642788887024, 0.374083548784256, -1.5700875520706177, 0.6193317174911499, 0.8095869421958923, -0.1929844319820404, -0.9597896933555603, -0.7610363364219666, -1.1136651039123535, 1.343109130859375, -0.28823739290237427, 0.5104517936706543, 0.6854014992713928, -0.7468589544296265, -0.8558025360107422, -1.7153136730194092, 0.42214301228523254, -0.7058895230293274, 0.5025916695594788, 0.7874857187271118, -0.24818652868270874, 0.7713644504547119, 1.0425610542297363, -0.02684655413031578, -1.9841176271438599, -1.1043533086776733, 0.8580037355422974, 0.5149469971656799, -0.3796384036540985, 1.2576465606689453, -1.1666607856750488, -0.7641733884811401, -2.4491682052612305, -1.5061054229736328, -2.0486679077148438, -0.29586464166641235, 0.21934133768081665, -1.9038716554641724, -1.8676763772964478, 1.8397024869918823, -1.8176106214523315, 1.0463358163833618, 0.5473983883857727, 0.12424912303686142, -0.8438248634338379, -0.38249897956848145, -0.4626668691635132, 1.2972862720489502, -0.6496660113334656, 0.6534355282783508, -0.1066732406616211, -0.5149866342544556, -1.1060954332351685, 0.8655242323875427, -0.6844170689582825, 0.13278284668922424, -0.751033365726471, 0.38466015458106995, 1.4263042211532593, -0.99736088514328, -0.3276299238204956, 0.4373154044151306, -0.4869413673877716, -0.8418483734130859, -0.08307614922523499, -0.35632264614105225, -0.5346167683601379, 0.9908292889595032, -0.6600242257118225, -2.669476270675659, -0.23102106153964996, -0.5847980380058289, -0.23677706718444824, -0.3724478781223297, 0.11520221084356308, 0.3564382493495941, -0.09379981458187103, 0.07866740971803665, -1.97239089012146, 0.9216582775115967, 0.8958202600479126, -0.08525717258453369, 1.8411153554916382, 0.962986171245575, -0.184797003865242, -0.8389765620231628, -1.1341526508331299, -0.2456476241350174, 0.04616212099790573, -0.628546416759491, -1.4126838445663452, 0.9829162955284119, 1.0645259618759155, -0.43625327944755554, -0.03154982253909111, 0.03049260564148426, -0.27153679728507996, -0.08268354833126068, -1.7816388607025146, 1.3673254251480103, 0.8296648859977722, -0.29043126106262207, -0.011512667872011662, 0.529071033000946, 0.10965263098478317, 0.05505140870809555, -2.239046096801758, 0.08778674155473709, -0.8527299761772156, 0.5747604370117188, 0.5375353097915649, -1.2065502405166626, -0.4010932147502899, -0.014315767213702202, 0.6816285252571106, 0.34117475152015686, 0.27445995807647705, -1.486724853515625, -1.9918239116668701, 0.6544832587242126, 0.7944399118423462, -0.1020294725894928, 1.3281124830245972, -0.6409286856651306, -0.08290165662765503, -0.6640335917472839, -1.2724130153656006, -0.12153364717960358, 0.08654029667377472, -1.2668659687042236, -1.0545852184295654, -1.1370424032211304, -1.2971627712249756, -1.3433765172958374, -0.25092390179634094, 2.889230728149414, 0.3705677390098572, 0.661225438117981, 1.2326350212097168, 0.8252573013305664, 0.12961940467357635, 0.4228554666042328, -1.23996102809906, 1.1312094926834106, -0.11060990393161774, 0.7597759962081909, 1.9695669412612915, -0.5861871838569641, 0.879082977771759, 2.4458255767822266, -0.32267287373542786, 0.1285354197025299, -1.1210569143295288, -2.1282293796539307, 0.05185380578041077, -1.626299500465393, 0.1453179121017456, -0.4414404332637787, -1.1401793956756592, -0.7910428047180176, 1.2230786085128784, -0.04881669580936432, 0.03719821944832802, -0.14374957978725433, 1.023655652999878, -1.2527598142623901, -0.61244136095047, -2.0678181648254395, -0.3724002540111542, -0.6118319630622864, -0.6677922606468201, -0.5602592825889587, 0.6964094042778015, -0.33202993869781494, 0.9967494010925293, -0.8856622576713562, 0.24181200563907623, 0.22286158800125122, 0.20767666399478912, -1.0076546669006348, -0.06711065024137497, -0.18498092889785767, 1.0037484169006348, -0.4560283422470093, 0.9409146308898926, 0.6266823410987854, 1.712437629699707, -0.5409597158432007, -0.606022298336029, 1.7841317653656006, 0.016439497470855713, -1.7559632062911987, 1.5373140573501587, 1.4320539236068726, 1.688594937324524, -0.3521866500377655, -0.913094699382782, -1.264510989189148, 0.41382381319999695, -0.645141065120697, -1.1806600093841553, 0.08916889876127243, -1.7262715101242065, -0.1797446459531784, 0.3048593997955322, 1.5979894399642944, -0.8373427987098694, -0.4712536633014679, 1.4835093021392822, -0.7840875387191772, -0.32935985922813416, 0.4140225350856781, -0.7997756600379944, -0.6095542907714844, -1.397325038909912, 0.34669163823127747, -2.873981475830078, 1.3416178226470947, -0.256929486989975, -0.09589489549398422, 1.3671084642410278, -1.8673022985458374, 0.595660924911499, -0.025386400520801544, -1.6982513666152954, -0.185503751039505, -2.0301809310913086, 2.0232348442077637, -0.6624468564987183, 1.3821054697036743, -0.816982090473175, -0.0858384519815445, -0.06397037208080292, 2.3594088554382324, 0.13409805297851562, 0.0483483262360096, -1.2537651062011719, 0.1769055873155594, -1.5735405683517456, 0.2197275608778, -0.178283229470253, -0.8513002395629883, -1.1629949808120728, 1.3579087257385254, -0.8561185598373413, -0.27315860986709595, 0.609806478023529, 0.3893541097640991, -1.1807526350021362, -0.556238055229187, -0.21410636603832245, 0.6409122347831726, 0.5484279990196228, 0.6126286387443542, -1.5710567235946655, -0.3021657466888428, 0.4120680093765259, -0.5137170553207397, 0.25064805150032043, 1.2619606256484985, -1.0507372617721558, 0.24809573590755463, -1.660233736038208, 0.740006685256958, 0.7188369631767273, -0.3386894464492798, -1.4518970251083374, -0.6896843314170837, 1.150638222694397, -0.05565683916211128, -1.7046890258789062, -0.3249218761920929, -2.000633478164673, 0.1413002461194992, 0.6258102059364319, -0.2815502882003784, 0.8046861290931702, -0.203848198056221, 0.7055038809776306, -1.1390352249145508, -0.26746705174446106, -1.2431104183197021, 0.6738772392272949, 2.4046733379364014, 0.7443287372589111, -0.8589091300964355, -0.3035643994808197, -0.25118494033813477, -0.45685651898384094, 0.3712178170681, -0.3989376127719879, 1.528821587562561, 1.443220615386963, -0.9949315786361694, -0.5449590086936951, -1.626879334449768, 0.9542443156242371, 0.4515763521194458, -0.5332175493240356, 0.5579705238342285, 0.4246237277984619, -0.1735018938779831, -2.8139801025390625, -3.1124532222747803, -0.12297706305980682, -0.13169167935848236, 0.805384635925293, 2.989365816116333, 0.20869103074073792, 0.3147031366825104, -1.2799121141433716, -1.4805952310562134, -0.4505821168422699, -0.07801226526498795, 1.5726882219314575, 0.7762106657028198, -0.5990731120109558, 1.1787245273590088, -0.5912654995918274, -0.9871902465820312, -0.391435444355011, -0.35507819056510925], [2.406482696533203, -2.0603654384613037, -0.6430536508560181, 1.9284303188323975, -0.21163271367549896, 1.0231853723526, 0.25670066475868225, 0.10487087070941925, 1.9324690103530884, 0.06635203212499619, 1.5053950548171997, -0.07225776463747025, -0.10963105410337448, -0.7536106109619141, 0.15827107429504395, 1.0940688848495483, 1.526176929473877, -0.09674502909183502, 0.4105934500694275, 0.8637261986732483, 0.7269513607025146, -0.6294148564338684, 1.0494840145111084, -0.646809995174408, 0.29331016540527344, 0.9111478328704834, -0.11404449492692947, -0.4995798170566559, -0.20559707283973694, -1.7142630815505981, 0.8789150714874268, -0.502687931060791, -0.6021339297294617, -1.901419997215271, -0.6748847961425781, -0.29092511534690857, 1.138131022453308, -1.115370512008667, 0.1264946609735489, 0.8216295838356018, -0.05348827689886093, 0.41356387734413147, -1.2245298624038696, 0.26505035161972046, 0.48238372802734375, 0.7992680668830872, -1.0071241855621338, 1.771468162536621, 0.8179274201393127, -0.987175464630127, 0.423496812582016, 1.4226690530776978, -1.2016963958740234, -0.05016015097498894, 0.17011018097400665, 0.981090247631073, 0.43747958540916443, -0.5569947957992554, -0.5203086733818054, 0.9053246378898621, -1.2188700437545776, -1.0959104299545288, 1.0872594118118286, 0.8384389877319336, 0.24264118075370789, -1.4978139400482178, -0.5237318277359009, -0.5626192092895508, -0.22004134953022003, -0.16025707125663757, -0.8334299921989441, -0.30807608366012573, -0.18332277238368988, 0.139393150806427, 1.172083854675293, 0.9338354468345642, 0.17510811984539032, -0.5957370400428772, -2.3201470375061035, -1.1161953210830688, -1.9988186359405518, -0.13227851688861847, -0.8959905505180359, 0.11131711304187775, 0.04054506495594978, -0.9089579582214355, -0.13748174905776978, -1.4312618970870972, 0.43368756771087646, 2.913166046142578, -1.6496343612670898, -1.585943341255188, 0.42249712347984314, -0.4253411591053009, -2.061405897140503, 0.5628678202629089, 0.04199037328362465, 0.16362154483795166, -0.010517660528421402, -0.23987585306167603, -1.6473265886306763, 1.1654698848724365, -0.025643104687333107, 0.14269733428955078, -0.491484671831131, -0.252914696931839, 0.18105819821357727, 0.7616629004478455, 1.6120200157165527, -2.6410319805145264, -0.660662055015564, -0.6761976480484009, -0.19189570844173431, -0.3710660934448242, -0.10661826282739639, -0.43961894512176514, 0.7720560431480408, 1.8053189516067505, 0.18109072744846344, 0.4502846598625183, 0.5828441381454468, -0.36639004945755005, -0.940208911895752, -0.5551962852478027, 1.046280860900879, 0.5430465340614319, -0.5091654658317566, -0.40692874789237976, -0.2497827708721161, 0.10420846939086914, -0.4064634442329407, 0.8107235431671143, -0.41225138306617737, -2.5298001766204834, 0.9562968015670776, -0.09882334619760513, -0.3499050736427307, -0.831042468547821, 3.0424118041992188, -0.026546338573098183, 0.8835088014602661, 0.2325367033481598, -0.28506967425346375, -0.5866300463676453, 1.0882568359375, 0.5113124847412109, 0.05581214278936386, 0.4726657569408417, 1.5447971820831299, 0.7182000279426575, -0.17288172245025635, 0.9058879613876343, 1.5779908895492554, -1.5859920978546143, 0.921818196773529, -1.6256693601608276, -0.4242969751358032, 0.8722741603851318, -0.23385801911354065, 0.19801746308803558, -0.37362679839134216, -0.03201625496149063, 0.011749071069061756, -0.7613643407821655, 0.5284757614135742, -0.18285033106803894, -1.7560735940933228, 1.9231995344161987, 0.8935201168060303, 0.08539699763059616, -0.9222895503044128, 2.4507601261138916, -0.7357590198516846, 0.8888483047485352, -0.0895656868815422, -1.498902440071106, -0.8305765986442566, -0.26547664403915405, -0.3585940897464752, 1.500891089439392, 0.0013613622868433595, 0.25230276584625244, -0.9054672718048096, -1.535035252571106, -0.3502851128578186, -1.0043741464614868, 0.22976675629615784, -1.1477971076965332, 0.2620723843574524, -2.2810757160186768, 0.444586843252182, -0.5051302909851074, 1.6671690940856934, 0.6020483374595642, -0.16942623257637024, 1.7587229013442993, 1.160167932510376, -0.5084066390991211, 1.0226186513900757, 0.6111653447151184, -0.5329291820526123, 1.976706624031067, 0.28985795378685, -0.44509169459342957, -0.05333547666668892, -0.22813430428504944, 0.8342514038085938, -0.07101716101169586, 1.5702314376831055, 1.3106341361999512, 1.6963632106781006, 1.4344948530197144, -1.8906214237213135, -1.3601024150848389, -0.8628726005554199, -0.5000035166740417, -1.029407024383545, -1.132872223854065, -0.45396819710731506, 0.3096165657043457, 0.316036581993103, -0.04443616792559624, -0.28626856207847595, 0.1567005068063736, -1.043945550918579, -1.4115530252456665, 0.8500713109970093, -1.414233684539795, -2.1901743412017822, -0.9357157349586487, 1.6629129648208618, -0.4244932532310486, -0.3828893303871155, -0.14950130879878998, 0.16550415754318237, -0.0523032508790493, -1.0750387907028198, 1.741809368133545, -0.8598572015762329, -0.5901201963424683, 0.539228081703186, 0.6591047644615173, 0.2802192270755768, -1.9126757383346558, 2.13987398147583, -0.8086351752281189, -0.916754961013794, -0.46388280391693115, 0.4491969048976898, 0.36451226472854614, 0.010966101661324501, 1.1607621908187866, 0.4455861449241638, 0.7999371290206909, 0.7407823204994202, -0.5319118499755859, 0.26971930265426636, 0.9843466877937317, 0.23775038123130798, -1.2823128700256348, 0.940980076789856, -0.8052002787590027, -0.2959193289279938, 0.956497073173523, -0.35594823956489563, -0.2455759197473526, 0.5254137516021729, 1.050326943397522, 0.3030197322368622, 0.5010275840759277, -0.8483439087867737, 1.5652028322219849, 0.5657323598861694, 0.7723308205604553, 1.938704013824463, 0.5684216618537903, 1.1670089960098267, 1.5764104127883911, 1.52615225315094, 0.15804290771484375, 0.055229511111974716, 0.43050944805145264, -0.5040473341941833, -0.31430625915527344, 1.3267134428024292, 0.12272123247385025, -0.5768098831176758, -0.06645696610212326, -0.038259994238615036, -1.3788784742355347, 1.0813997983932495, -0.6438825130462646, -0.3998074531555176, 0.8685061931610107, -0.689593493938446, 1.422512173652649, 0.1095237210392952, 0.47046831250190735, -0.07749976962804794, 0.5703541040420532, 0.9277051687240601, 2.0370326042175293, 1.1102969646453857, 0.5508576035499573, 0.34246817231178284, -0.3518037796020508, -0.13996857404708862, -1.6099885702133179, -1.1420209407806396, -0.028918558731675148, -0.7863053679466248, -0.6594520807266235, -0.07285069674253464, 0.3797767758369446, -1.38817298412323, -0.8732664585113525, 0.9628297090530396, -0.3311925530433655, 1.0679984092712402, 0.5140962600708008, 1.9014408588409424, -2.183985710144043, -0.7665237784385681, -0.4285334646701813, 0.16656605899333954, 0.2801986038684845, -0.22559596598148346, 0.12103493511676788, 0.09448956698179245, 0.11697088927030563, 2.0943820476531982, 0.671525776386261, -0.9552710056304932, 0.9418075680732727, 0.6798294186592102, 1.3293581008911133, -0.21986229717731476, -0.4233413338661194, -0.08667098730802536, 0.7961620688438416, -0.3559195101261139, -1.2595731019973755, 1.5214171409606934, -1.0918910503387451, 0.6925466656684875, 0.1653790920972824, 0.0738934874534607, 1.3336385488510132, 0.9152975678443909, 0.13835282623767853, -0.37489446997642517, 1.7662795782089233, -0.4913327395915985, 0.046094588935375214, -0.8116800785064697, -0.5518898963928223, -0.6733170747756958, -1.4387325048446655, -0.14429113268852234, 0.25340327620506287, -1.2792603969573975, 0.31435972452163696, 0.1794085055589676, -0.34435349702835083, 0.5689172148704529, -0.11991225183010101, 0.16197830438613892, 0.23329228162765503, 1.5217214822769165, -0.9010027050971985, 1.2929586172103882, 0.889787495136261, -3.3513453006744385, -1.0203707218170166, -1.4428682327270508, 1.7747278213500977, -1.2009755373001099, 1.2440941333770752, -0.4774773418903351, -0.4808330833911896, -0.7225494384765625, 0.315083771944046, 0.5786005258560181, -0.7476422190666199, -0.1802252233028412, 1.3793697357177734, -1.1515028476715088, -0.1283743530511856, -0.12778562307357788, -0.6388503313064575, 1.440197467803955, 0.8125934600830078, 0.5628505349159241, -0.13511046767234802, -0.6163851618766785, 0.0893908441066742, 0.00023837792105041444, -0.41370147466659546, 0.857515811920166, -0.2016417384147644, -1.0794752836227417, -0.11525911837816238, 0.33903220295906067, -0.6351912021636963, 0.6158657073974609, -1.2985739707946777, 0.486855149269104, -0.24649794399738312, -0.580436646938324, -0.10877831280231476, 0.2690449357032776, -0.7282310128211975, 0.5504173636436462, -0.23009325563907623, -1.3250967264175415, -1.239980936050415, 0.15711082518100739, 0.37091222405433655, 0.5655925869941711, -0.7457917928695679, 0.5537561774253845, 1.5829495191574097, -0.5569206476211548, -0.8443012833595276, 0.2003391832113266, -0.856375515460968, -0.7835906147956848, -0.4310480058193207, 0.602001965045929, -0.10569912195205688, -0.26006385684013367, 2.1422083377838135, -1.6901609897613525, -1.8955336809158325, -0.4107363224029541, -0.6466240882873535, -0.011352240107953548, -1.6432254314422607, -0.11317227780818939, 0.01264809537678957, -0.9724669456481934, 1.284137487411499, -0.9732667803764343, -0.957228422164917, 0.10674093663692474, 0.4790652394294739, -0.02181749977171421, 0.3187319338321686, -0.20610933005809784, -0.3396061062812805, 0.008987934328615665, 0.05531187355518341, 0.5388585329055786, 1.6343704462051392, 0.09908436983823776, 0.25028523802757263, 0.21961276233196259, -1.6099109649658203, 0.43903475999832153, 1.233030915260315, 0.5213972330093384, -0.2554773688316345, 0.8809766173362732, -0.5306355953216553, -2.2522385120391846, -0.3267628252506256, -1.6974376440048218, -1.3215396404266357, 0.4417571425437927, 0.02155446819961071, 0.3842175304889679, 2.542020559310913, -0.7909091711044312, 0.24687835574150085, 0.4428754150867462, 1.17763090133667, 0.9999770522117615, -1.0049301385879517, -1.2090190649032593, 0.5906779170036316, -0.21188358962535858, 0.09493906795978546, 0.2713073790073395, -0.9499427080154419, 0.06368428468704224, -0.8719355463981628, 0.10904181003570557, -0.8531463742256165, -0.6144589781761169, 0.5462197065353394, 0.6551300287246704, -1.2473689317703247, -0.6148092746734619, -0.42138704657554626, -0.33310356736183167, 0.4167329668998718, -0.2285172939300537, 1.2709064483642578, -2.0219483375549316, -0.758097767829895, 2.210296869277954, -0.5110392570495605, -1.375921607017517, 0.9292109608650208, -0.9286872148513794, 0.06291701644659042, -1.6263452768325806, 0.4954303205013275, 0.7602190375328064, 0.626457691192627, 0.7395824193954468, -1.1836761236190796, 0.574000358581543, -1.3576934337615967, 0.8366362452507019, 1.167270541191101, -1.3075426816940308, 0.37715479731559753, 1.1283069849014282, 0.03974384069442749, -0.5890045166015625, 0.8668290972709656, -0.25323671102523804, 0.8367449641227722, -0.19691234827041626, 0.2037629336118698, 0.24181941151618958, 1.2648746967315674, -1.81385338306427, 0.6583917140960693, -0.9352765679359436, 0.8432605266571045, -2.2615725994110107, 0.26724207401275635, -0.4308265447616577, 1.150349736213684, -0.5227506160736084, 1.104897141456604, 0.22453345358371735, 0.7851631045341492, -0.1677524596452713, 0.677423894405365, -0.40762144327163696, 2.0650405883789062, 0.09022971987724304, 0.19683557748794556, -0.19049032032489777, -0.6021803617477417, 0.19031603634357452, -1.9449294805526733, 0.46782439947128296, -0.5466129779815674, 0.05566106364130974, -0.4762895107269287, 1.318116307258606, 0.24745479226112366, 1.2701619863510132, 1.4408564567565918, -0.455206960439682, -0.6406584978103638, -0.6331024765968323, -0.907996654510498, 0.05841165781021118, 1.154847264289856, -1.4579555988311768, -1.2846647500991821, -1.8417418003082275, -2.4050304889678955, -1.754549264907837, -0.053974028676748276, -0.7949495911598206, -0.36001700162887573, 0.04293685033917427, 0.16713927686214447, -0.2321360558271408, 0.16370926797389984, 1.1393955945968628, 0.9115800857543945, -2.1784374713897705, -0.7956653833389282, 0.7844840884208679, -0.009916235692799091, 0.8302468657493591, -0.2791403532028198, 1.5357645750045776, 1.5383479595184326, 0.42241328954696655, -0.853796124458313, 0.9312651753425598, -0.007473495323210955, -0.023544084280729294, 0.6777069568634033, 0.45280495285987854, 0.9085120558738708, -0.0013746499316766858, 0.2915545701980591, 0.38060927391052246, 0.46518686413764954, 0.46526801586151123, -2.380324125289917, 1.5531721115112305, -0.8014689087867737, -0.5830128788948059, -0.9758803844451904, 0.5180810689926147, 1.0464975833892822, 1.4387376308441162, 1.980196475982666, 0.20450569689273834, -2.0278162956237793, -0.6033056378364563, -1.1549137830734253, 0.05553685501217842, 0.031140627339482307, -1.8732080459594727, -0.25611892342567444, -0.09281609207391739, -1.082361102104187, 1.708607792854309, 0.7176147699356079, -1.0887473821640015, -0.30473554134368896, -1.4420868158340454, -0.21730464696884155, -1.0644680261611938, 0.3937532901763916, 0.37518689036369324, -1.1612657308578491, -1.1657977104187012, -0.43759262561798096, 2.3042445182800293, 0.6981449723243713, -0.5494246482849121, -1.1498494148254395, 1.2037125825881958, 0.08182967454195023, -0.6357437372207642, 0.1141737550497055, -0.3334932029247284, 0.14307284355163574, -0.9980488419532776, -1.7686926126480103, -0.2110861837863922, -0.22121046483516693, 0.8657817840576172, 1.323776364326477, -0.8162117004394531, 2.2922170162200928, 0.18393269181251526, -2.3566160202026367, -0.08223656564950943, -0.39370694756507874, 0.043127018958330154, -1.1159788370132446, -1.5189054012298584, -1.0357791185379028, -0.6105819940567017, 0.9148412942886353, 0.2997717261314392, 1.223311424255371, 0.48821720480918884, 0.9414412379264832, -1.2842503786087036, 0.977986216545105, -1.699205994606018, -1.5937914848327637, 0.10929135978221893, -0.24706993997097015, -0.010050258599221706, -0.38404831290245056, 0.5598746538162231, 1.0267153978347778, -0.8331685662269592, 1.158301591873169, 0.43629246950149536, -1.0615649223327637, -0.7112554907798767, 0.37603798508644104, 0.29368382692337036, -0.3775874972343445, -0.8067395687103271, 0.02633410319685936, 1.3633865118026733, -1.0081379413604736, 0.5001762509346008, -0.44311630725860596, -0.8206226229667664, 0.9792572259902954, 0.03612595796585083, 0.20351779460906982, -0.06199183315038681, 0.1779116690158844, -0.36703407764434814, -0.7154597043991089, 1.025317668914795, -1.1489535570144653, 0.16729819774627686, 2.689011573791504, -2.1275603771209717, 0.4877197742462158, -0.9793787002563477, 0.605266809463501, 1.6522068977355957, 0.34948861598968506, -0.5639312267303467, -0.46393340826034546, 0.4140494167804718, 0.41487544775009155, 0.6902297139167786, -0.49969398975372314, 0.1932772845029831, 0.3626265823841095, -1.0525468587875366, -0.2933425009250641, 0.9402063488960266, -1.2534464597702026, -0.19687199592590332, 0.6616843342781067, -0.9767405986785889, 0.3423463702201843, 0.6018055081367493, -0.8871321082115173, 0.13981087505817413, 0.5864824056625366, 0.5011909604072571, -1.4085944890975952, 0.6852436065673828, 0.5537845492362976, -1.0687464475631714, 0.6837696433067322, -0.4509149193763733, 1.3627370595932007, -0.5067124962806702, 1.9082286357879639, 1.0199116468429565, 0.38791343569755554, -0.7820665240287781, -1.2445590496063232, -3.2525830268859863, 0.33198997378349304, 0.507565438747406, 0.6026551723480225, -1.5827397108078003, 0.7484655976295471, 1.1132113933563232, -0.8829410672187805, 1.4156227111816406, -0.5902548432350159, 0.6453940272331238, -0.47053003311157227, -1.448097825050354, -1.1939564943313599, -0.07999256253242493, 0.489554226398468, 0.8375604748725891, 2.579624652862549, 0.44562485814094543, 0.986969530582428, -1.8085306882858276, 0.4485842287540436, 0.5545547008514404, 0.6179163455963135, 1.4983290433883667, 0.46781468391418457, 1.1809715032577515, -1.1231435537338257, 0.1572229117155075, 0.5612760186195374, -0.11817894876003265, -1.5603173971176147, 0.0515117347240448, -0.027800440788269043, 0.4321308732032776, -0.9841704964637756, 1.0937318801879883, 0.8497177362442017, 0.3275464177131653, -0.06966487318277359, 0.6395193338394165, 0.8036302924156189, -0.8144956231117249, 0.2270762026309967, 0.49534982442855835, -0.02946842648088932, -0.781779408454895, -1.3826117515563965, -0.6102055311203003, 0.32267898321151733, -0.27395009994506836, -0.4533734917640686, 1.185460090637207, 0.8771193027496338, -0.14992301166057587, 0.3102576732635498, 0.9614440202713013, -0.6380727291107178, 2.026273727416992, -0.7593923211097717, 0.1654340922832489, 2.3912854194641113, 0.6782879829406738, -1.3471342325210571, 0.696843147277832, -0.9938929080963135, -1.94614577293396, -1.2790987491607666, -1.107747197151184, -0.5635380744934082, 0.6323234438896179, -0.8413278460502625, -0.06547237932682037, 1.0467579364776611, 0.4247455298900604, 0.6716671586036682, 0.48694440722465515, 0.6834697127342224, 0.6476892828941345, 2.0280673503875732, -0.05792030692100525, 0.28456932306289673, 0.9408430457115173, -0.16059590876102448, 0.9785059690475464, 0.7870467305183411, 0.7157257795333862, 0.6475218534469604, 0.2611069977283478, 0.44850072264671326, 0.8260402083396912, -0.10556066036224365, -0.3752903938293457, -0.48031583428382874, 0.6472393870353699, -1.7202335596084595, 1.6668753623962402, -0.5941903591156006, -0.4050159156322479, 0.33473777770996094, -0.7266010642051697, -1.4731192588806152, -1.6483646631240845, -0.3333923816680908, 1.6134659051895142, 0.29484838247299194, -0.2999865710735321, -0.2691406309604645, -1.8429590463638306, 1.155429720878601, 2.098687171936035, 0.22140155732631683, -0.14077246189117432, -0.2145298570394516, 0.7231918573379517, -0.9115577340126038, -0.7486224174499512, -0.566127359867096, 0.9378401637077332, 0.020253635942935944, 1.3073946237564087, 1.2524492740631104, 1.0861623287200928, -0.28611502051353455, 0.6489344239234924, 1.6576818227767944, -0.08011522144079208, 1.113347053527832, 0.5651256442070007, 0.1296241581439972, 0.18308325111865997, 0.002272882265970111, 0.3089229166507721, -1.824066400527954, -0.5846943855285645, 0.1416676938533783, -0.987798810005188, -0.9016727805137634, -0.4987104535102844, 0.3871319890022278, 2.021577835083008, -0.8111697435379028, -0.5772882103919983, 0.1737065613269806, 1.0398000478744507, -1.2576321363449097, 0.8891903162002563, -0.13702645897865295, 1.0606346130371094, -0.13305528461933136, -0.10454609990119934, 0.6233670711517334, -0.9202222228050232, 0.3983655869960785, 0.23753418028354645, 0.3582684397697449, 0.1116594448685646, -0.5834169387817383, -0.6212085485458374, -0.34349504113197327, -0.3361099362373352, -1.0102919340133667, -1.8355085849761963, 0.6045753359794617, 0.33095845580101013, 0.783081591129303, -0.18415850400924683, -3.474043846130371, -0.018914202228188515, -1.5573241710662842, 0.5293063521385193, 1.4289523363113403, 0.4540017247200012, 0.5209762454032898, 0.3513382077217102, -1.1035516262054443, -0.19378654658794403, 0.8284893035888672, -0.0735790804028511, 0.31715139746665955, -0.1361786276102066, 0.7847703099250793, 0.5620067715644836, 0.06915290653705597, -0.11727949231863022, 3.3661458492279053, 0.27948248386383057, -0.10767438262701035, 0.22016216814517975, -0.002837373875081539, 0.6504542827606201, -0.6176270246505737, 1.3206385374069214, -0.5788094401359558, -0.8561037182807922, 0.4454110860824585, -0.5291184782981873, 1.2041693925857544, 1.814748764038086, -1.2817249298095703, 0.4990350008010864, -0.7481991648674011, -1.0828158855438232, 0.09212314337491989, -0.9486268758773804, -0.2385001927614212, 0.10022015124559402, -2.300985336303711, -2.1146228313446045, 0.12361755967140198, 0.8237810730934143, -0.9824573993682861, -0.37990209460258484, 0.4771426022052765, 0.5826188921928406, 1.5675276517868042, 2.850642681121826, 0.4576911926269531, 0.861054539680481, -0.09245499968528748, -0.4544414281845093, 0.2899360954761505, 0.6239981651306152, 2.604041576385498, -1.3689955472946167, 1.2004015445709229, -0.5245428085327148, -0.4783758819103241, -1.5326488018035889, -0.831637978553772, 0.2630150318145752, 0.1968860924243927, 1.0608625411987305, -1.3172991275787354, -0.613028883934021, -0.6173369288444519, 1.0532065629959106, 0.45256054401397705, 0.005639227107167244, 0.6643117666244507, -1.535393238067627, -0.17898568511009216, -0.9871571660041809, -0.10812420397996902, 0.468558669090271, -0.7533578276634216, 1.6517664194107056, -0.9474704265594482, 0.2410183846950531, 0.10856204479932785, 0.23782412707805634, 0.9281381368637085, -0.4550195634365082, 0.35651588439941406, 0.3121495544910431, -0.5733538269996643, -1.2080124616622925, -0.4823940098285675, 0.5787999033927917, -1.3694850206375122, -0.2296520620584488, -1.241721749305725, -0.18518120050430298, -0.8872342109680176, -0.012219741009175777, 1.116755723953247, -0.2618831992149353, 0.4635794758796692, -0.46698349714279175, 1.759883165359497, -0.9826496839523315, 0.7336645722389221, -0.720573365688324, -0.04367918521165848, 0.3977297246456146, -0.1678665727376938, 0.26529043912887573, -1.2734639644622803, 1.6242834329605103, 1.2092392444610596, 0.9686129093170166, 0.022479599341750145, 0.6147342324256897, -1.468355655670166, -1.4175541400909424, 1.6516849994659424], [0.05605260655283928, -0.19753971695899963, 0.841098427772522, 0.683184802532196, 0.3819328546524048, -1.0091500282287598, -0.9831851720809937, 0.49496138095855713, -1.1119389533996582, 0.13986241817474365, 1.6306778192520142, -1.3722410202026367, -2.6286284923553467, 0.6108064651489258, -1.0145989656448364, -0.17219702899456024, 0.16936248540878296, -0.4967212378978729, 0.36806777119636536, -1.4047250747680664, -0.058095913380384445, -1.1696487665176392, 1.0890365839004517, 1.626625895500183, -1.2787282466888428, 0.11553765088319778, -0.17547331750392914, -1.0607224702835083, 0.8164351582527161, 0.9166945219039917, 1.1278085708618164, 0.8847711682319641, -0.7487692832946777, -0.32825860381126404, -0.0503101646900177, 0.34881091117858887, 0.44245851039886475, -1.7138822078704834, -0.1933179497718811, 0.8039491772651672, 0.5549856424331665, 0.7230424880981445, 1.9969080686569214, -0.3014293313026428, 1.1045782566070557, 1.458219289779663, -1.077658772468567, -2.9015960693359375, 0.28878191113471985, 2.1704957485198975, 0.09680495411157608, 0.7909533381462097, -0.6433175206184387, 0.10766290873289108, -0.6629364490509033, 0.05864781141281128, -0.9590388536453247, -1.5927404165267944, 0.05914933979511261, 0.18489739298820496, 0.5650618672370911, 0.6185333728790283, 0.48124146461486816, -0.6420714855194092, 1.9279115200042725, 0.09506043046712875, -0.02651190012693405, -2.1845498085021973, 1.745609164237976, 0.5234054327011108, -1.0379904508590698, 0.1880800575017929, -0.2142704427242279, -0.5709165334701538, -0.5908286571502686, -1.9667850732803345, -0.5285293459892273, 1.4080365896224976, 0.13753756880760193, 0.19414474070072174, 0.9581350684165955, 0.7162185907363892, -1.0121220350265503, 1.304091215133667, 0.006542655639350414, -0.4057409465312958, 1.1238187551498413, 0.7011400461196899, 0.5006705522537231, 1.4507488012313843, -0.47553539276123047, -1.6467258930206299, -0.35805290937423706, 1.2934879064559937, -0.7391635179519653, 1.1743501424789429, 0.16143415868282318, -0.4087834358215332, -0.01835343800485134, 2.507835865020752, 1.3833088874816895, 0.5445908308029175, 0.42449522018432617, -0.011912466026842594, 0.09038930386304855, -1.2997381687164307, -0.806167483329773, -0.7598507404327393, 0.20284557342529297, -0.8492151498794556, -0.6825512647628784, 0.46283960342407227, -1.6166093349456787, 0.10723526775836945, 0.20238378643989563, -0.44685640931129456, -0.7536337375640869, 0.5707019567489624, -1.7517319917678833, -0.572930097579956, -0.7028557062149048, -0.03620945289731026, 0.5855937004089355, -0.21757183969020844, -0.6586211919784546, 0.4542918801307678, -1.3076485395431519, 0.6577459573745728, -0.11846312880516052, -0.7330062389373779, 0.6528921723365784, -0.5438385605812073, -0.05859723687171936, -1.043481469154358, -0.6752730011940002, -1.7809817790985107, -0.715175449848175, -1.3320766687393188, 0.6991820335388184, -0.037103794515132904, 1.6410330533981323, -0.035494569689035416, -0.13852640986442566, -0.5639053583145142, -0.6846452951431274, 0.207819864153862, 0.035920094698667526, -1.6242029666900635, 0.672941267490387, -0.12686367332935333, -0.625511646270752, -0.5987448692321777, -0.27969592809677124, -0.1110018864274025, 1.0282777547836304, -0.7016612887382507, -0.7599126696586609, -1.8782833814620972, -0.40322741866111755, -2.396021604537964, -0.9138184785842896, 0.036253344267606735, 0.23197978734970093, 0.11707571148872375, -0.38857418298721313, 1.063393473625183, -1.994750738143921, 0.021818675100803375, 0.056440211832523346, -1.324562668800354, -0.02175184153020382, -0.7572223544120789, 1.7086178064346313, -0.7501964569091797, 1.1162734031677246, 0.0038817464374005795, -0.2718179225921631, -0.6704655289649963, -0.20030494034290314, 0.7566343545913696, -0.47731825709342957, -0.3975817859172821, 1.1339657306671143, 0.8219972848892212, -0.5208410024642944, -0.6887528896331787, -0.9001453518867493, 1.3242703676223755, -0.14505413174629211, 0.7837328910827637, 0.4612700939178467, 0.9307869672775269, 0.010869654826819897, 0.7946261763572693, 0.6197618842124939, 0.1996973603963852, -0.18157780170440674, 0.4308842420578003, 2.8406906127929688, 0.1516066938638687, 0.23262344300746918, -2.0353612899780273, -0.7502968311309814, 1.1819419860839844, -0.9705703854560852, 1.1905702352523804, -1.5005491971969604, 0.3244918882846832, 0.32195454835891724, -0.041281621903181076, 0.24803543090820312, -0.1389310359954834, 1.5628467798233032, -0.8283653259277344, -2.1588969230651855, 1.5082077980041504, 0.5489607453346252, 1.7295976877212524, 0.6490100622177124, 0.2767599821090698, -0.9628416895866394, -0.38219204545021057, -0.1431330293416977, 1.0368486642837524, -0.7970675826072693, 1.055633783340454, -0.9265689849853516, -0.4013359546661377, -0.9716182947158813, -0.9032306671142578, -0.4655720293521881, -0.510448157787323, -0.3600856363773346, 0.3097805380821228, -0.7597906589508057, 1.4538525342941284, -0.7072740197181702, -0.18125173449516296, -0.3861004114151001, -0.43965214490890503, 1.6363770961761475, -0.33154913783073425, -0.37184908986091614, -1.301924228668213, 0.5873051285743713, 0.7464421391487122, 1.6521410942077637, -0.7393519878387451, -1.6622655391693115, 1.587365984916687, -0.6562681198120117, -0.6900162696838379, -0.7307178378105164, 1.0857080221176147, 0.4000420868396759, 1.607590913772583, 0.055834002792835236, -0.464324414730072, 0.5224089026451111, -0.5213677287101746, 1.0721094608306885, -0.5919585227966309, -1.3496131896972656, -0.030075136572122574, 0.2211918979883194, -0.49894988536834717, -0.22812552750110626, 0.45555922389030457, -1.8565086126327515, -0.2524106502532959, -0.8924422860145569, 0.04546216130256653, 0.6929213404655457, 1.1982293128967285, 0.9257575273513794, 0.3821927011013031, 0.0840071365237236, 0.30933457612991333, -1.0644285678863525, 1.7370809316635132, 0.6724212169647217, 0.39683955907821655, -1.314027190208435, -0.3911588788032532, 0.7993832230567932, 0.019122816622257233, 0.6407713294029236, -1.38670015335083, 1.1445426940917969, -0.11107946932315826, -1.3492121696472168, -0.13296382129192352, 0.14538028836250305, 0.7480708360671997, -3.418168783187866, -2.345695972442627, -0.27462849020957947, -0.6769623160362244, 0.4092659652233124, -0.36092063784599304, 0.07104308158159256, -1.7079389095306396, 1.0339982509613037, -0.9410548210144043, 0.0038382478523999453, -1.3531296253204346, -0.03295031934976578, 0.36520859599113464, 0.10234382003545761, -1.5637956857681274, -1.3944690227508545, -0.5022020936012268, 0.37071314454078674, 0.9971681237220764, -1.5067354440689087, 0.30524390935897827, 0.0553508922457695, -0.34990301728248596, -0.25273996591567993, 1.757706880569458, -1.2221087217330933, 0.5182680487632751, -0.5127604007720947, 0.4139135479927063, 0.6827279925346375, -1.355423092842102, -0.3802151679992676, -0.6260266900062561, -0.06034044548869133, 0.18694333732128143, -2.118009567260742, -0.24076351523399353, -1.069536805152893, 0.7409191131591797, 0.45267581939697266, -1.2215073108673096, -0.5367404818534851, -0.27770528197288513, 0.08667797595262527, 0.2803433835506439, 2.415738344192505, -0.18633252382278442, -0.846214234828949, -2.0792043209075928, -0.9400685429573059, -0.8624874949455261, 0.5282512307167053, -0.09243061393499374, -0.3590559959411621, -0.015124065801501274, 0.14096006751060486, -2.219888210296631, 0.6321127414703369, 0.7971476912498474, -0.9731541275978088, 0.3705732822418213, -0.9658389687538147, 0.6979123950004578, -0.35923922061920166, 1.2440978288650513, -1.9806091785430908, -0.9598453640937805, -0.023957202211022377, -1.1457645893096924, -1.1897250413894653, -0.3282080590724945, -0.09003373980522156, -0.2690882682800293, -0.808204710483551, 1.126525640487671, -1.2537946701049805, 1.6050525903701782, -1.7346099615097046, 1.1030538082122803, -0.7899211049079895, -1.1960755586624146, -0.411832720041275, -0.9175777435302734, -0.48356059193611145, 0.3869358003139496, 1.1277036666870117, -0.12341710180044174, 0.39678433537483215, -1.3859108686447144, -0.06920188665390015, 1.2362728118896484, -0.5787070989608765, 1.1793006658554077, 0.14130577445030212, 0.29560473561286926, 0.6532553434371948, -1.7312721014022827, 0.7092788815498352, 0.1248413473367691, 1.421402096748352, -0.9225725531578064, 0.4776794910430908, 0.5232395529747009, 0.7349029183387756, 2.5389087200164795, -1.3653076887130737, 0.47327229380607605, -0.4909771978855133, -0.6989907622337341, 0.2332438975572586, 1.126086950302124, 0.3326959013938904, -0.255565881729126, 0.8748674988746643, 0.3899868130683899, -0.011293668299913406, -0.49815061688423157, 0.8596255779266357, 0.17176589369773865, 1.2121365070343018, 0.33546096086502075, 2.401672124862671, 1.4567888975143433, 0.4675399661064148, -0.7756116390228271, -1.4953690767288208, 1.0438815355300903, 0.2526770234107971, 0.9254072904586792, 0.925089418888092, 1.3445498943328857, 1.2676808834075928, 1.821076512336731, 1.8485796451568604, -0.27366119623184204, 0.5402427911758423, -0.9438809156417847, -0.5455648303031921, -1.377629280090332, 2.4769794940948486, 0.02412230707705021, -1.175233006477356, -1.1501233577728271, 0.5444868206977844, -0.8751537203788757, 1.0712615251541138, 0.609473705291748, -1.1768434047698975, -0.8440777063369751, -0.4409641623497009, -1.5094674825668335, -1.3668458461761475, 0.042303431779146194, -0.7605887055397034, 0.833946943283081, 0.015251364558935165, -0.8716263771057129, 0.4202190339565277, -0.5129137635231018, 0.5717688798904419, -2.029735565185547, -1.4134390354156494, -0.4868648648262024, -0.07765135914087296, 1.8977365493774414, 0.6426486968994141, -0.8372129201889038, -1.9345929622650146, -0.09578414261341095, -0.176676407456398, -1.4229458570480347, -0.19730983674526215, -1.4230870008468628, 0.15831084549427032, 0.4468226730823517, -0.5501468777656555, -0.10821579396724701, -2.182398557662964, -0.4356001019477844, 0.06556212157011032, -1.8264707326889038, 0.20112445950508118, 0.12466031312942505, 0.26744601130485535, 0.614732027053833, 0.2563623785972595, 0.5904803276062012, 0.42099785804748535, -0.18176651000976562, 1.4550355672836304, 1.7837649583816528, -1.5160161256790161, -0.04546922817826271, -1.2832939624786377, -0.7452880144119263, -2.4804530143737793, -0.0693325474858284, 0.5821007490158081, 0.12208675593137741, -0.2849976122379303, 0.5741721391677856, 0.28705161809921265, -0.9163126349449158, 0.6351004838943481, -0.6833792924880981, -1.6341439485549927, -0.3056737184524536, -1.372054934501648, 0.6006990075111389, 0.22662536799907684, 0.4978741705417633, -0.8334607481956482, 0.22650913894176483, 0.9389116168022156, 0.616102397441864, 0.8080464601516724, -0.5419516563415527, -0.8139804005622864, 1.0017777681350708, 0.1420954316854477, -2.1706950664520264, -1.3641194105148315, 0.8946669697761536, 0.023246776312589645, 1.2694919109344482, 0.6977947354316711, 0.3601435720920563, -1.4080426692962646, 0.7472769021987915, 0.26507750153541565, -0.09992051124572754, -1.0144774913787842, 0.5911499857902527, 0.2886240780353546, 1.280948281288147, 1.293372392654419, -0.20640671253204346, -0.10972854495048523, 0.7322360873222351, -1.2177776098251343, -0.515304446220398, -0.05391534045338631, 1.8025468587875366, -0.27202415466308594, 0.6467235684394836, 0.15822142362594604, 1.213931918144226, 0.8845617771148682, -0.5987260937690735, -1.2122795581817627, 0.18680532276630402, 1.2467650175094604, -0.7816120982170105, 1.01287841796875, 0.007489145267754793, 0.868812084197998, 0.6585961580276489, 0.10550225526094437, -0.9190992116928101, -0.13308484852313995, 0.2371455430984497, 0.015167893841862679, -0.055971015244722366, -2.287623643875122, -0.20924201607704163, 0.45640474557876587, 1.0690345764160156, 1.1602178812026978, 0.24684277176856995, 1.256629228591919, -0.5845458507537842, 0.23266752064228058, 0.9033300280570984, 0.3914628028869629, 0.721240758895874, -0.351799339056015, 0.35600802302360535, -1.4859532117843628, 1.4341340065002441, 0.5271641612052917, -0.7550939917564392, -0.7350820899009705, -0.2189565896987915, 0.30191367864608765, -1.3145920038223267, -0.24285617470741272, -0.7516464591026306, 1.182693362236023, 0.8090542554855347, 0.36014339327812195, -0.19654174149036407, 1.1174671649932861, 1.750288486480713, 0.1406720131635666, -0.8545143008232117, -0.020371967926621437, 0.21789143979549408, -1.5297120809555054, 0.35852545499801636, -1.0713322162628174, 0.36240875720977783, 0.5463506579399109, 1.1726802587509155, 0.3202274441719055, 2.026555061340332, 0.8083891272544861, 0.3491153120994568, -1.0040843486785889, 2.4234535694122314, -0.015024730935692787, -0.35012751817703247, 0.6545670628547668, -0.6297570466995239, 0.6097110509872437, -0.8294951915740967, -1.0408202409744263, -2.9567317962646484, 1.3046326637268066, 1.4637788534164429, -2.2513604164123535, 1.476202368736267, 0.5718302130699158, -0.19185420870780945, 1.3333319425582886, -1.0010161399841309, 2.028780698776245, 0.583486795425415, -2.3254284858703613, -0.34715431928634644, 0.09308478236198425, -0.834732711315155, -0.4394591748714447, 0.256430447101593, 0.62799471616745, 0.42612922191619873, 0.5582191944122314, 0.02834552899003029, -0.5152038335800171, 0.6845998167991638, -0.12400887161493301, 0.6073132753372192, 0.706081211566925, -0.39681074023246765, -0.14273974299430847, 0.46135246753692627, -0.009555293247103691, 1.7984639406204224, -0.5222796201705933, 1.3963357210159302, -0.49371781945228577, -0.4347495138645172, -0.07792896777391434, 0.376248300075531, -2.198193311691284, -1.2983378171920776, 0.11704432964324951, -0.4136296808719635, -0.20078954100608826, 0.8694552183151245, -0.5839786529541016, -0.04528562352061272, 0.9974563121795654, 0.29308485984802246, 1.2042367458343506, 1.119844913482666, -0.783094584941864, -1.1772007942199707, 1.2025409936904907, -0.6878716945648193, 0.7651513814926147, -0.4982796609401703, 0.04748456925153732, 0.4111528694629669, -1.4267274141311646, -0.3127531111240387, -0.04156990721821785, 1.3128142356872559, 0.31954899430274963, 0.5527623295783997, -0.8258639574050903, 0.9869364500045776, 1.2002015113830566, 0.18350380659103394, 0.7717879414558411, -1.1302212476730347, -0.12829433381557465, 0.8084871768951416, -1.0892726182937622, 0.6881855726242065, 0.582004725933075, 1.7371407747268677, 0.5404074192047119, 0.6504673957824707, 1.8051553964614868, 1.0377721786499023, -0.7309620380401611, -0.4366523027420044, 0.13975101709365845, -0.7354657053947449, -0.4491925537586212, -0.9927836656570435, 1.640477180480957, 1.046321988105774, -0.24789254367351532, 0.8161109685897827, 0.5164299607276917, 0.4675738215446472, 0.43329480290412903, 1.3213090896606445, 0.4358425736427307, 0.11660691350698471, -0.5826551914215088, -0.4238690733909607, -0.08390957117080688, 0.5979610681533813, 0.6841307282447815, -0.5397506356239319, -0.6635660529136658, 0.21504656970500946, -1.1249721050262451, -0.5831857919692993, 0.25746950507164, -0.5792127251625061, 0.6116766333580017, -0.12596002221107483, -0.31661081314086914, 0.34112846851348877, -0.4973240792751312, -0.43975475430488586, 1.0838632583618164, 0.6514987945556641, -2.309194326400757, 0.4595005512237549, -1.3336637020111084, 2.1677963733673096, 0.06262067705392838, 0.13692276179790497, -1.1104412078857422, 0.38839206099510193, -0.03837960585951805, 0.4960094094276428, 0.7237744927406311, -0.33593228459358215, -0.5444096326828003, 2.5980308055877686, -1.1696218252182007, -0.0014423844404518604, -0.9464481472969055, 0.17969703674316406, -0.5539384484291077, 1.0049278736114502, -0.3570702075958252, 0.5077693462371826, 0.43103572726249695, -1.0206204652786255, -0.5526320934295654, -0.7931097149848938, -0.18155114352703094, 0.3066239058971405, -0.9803267121315002, -0.4943080246448517, 0.6060094237327576, 0.7479561567306519, 1.1752128601074219, 1.3296865224838257, 1.079228162765503, 1.283892035484314, 1.5547599792480469, -0.5940324068069458, -0.04071645438671112, -1.0042927265167236, -0.17508675158023834, -0.37167826294898987, 1.3619953393936157, 0.37227728962898254, -1.2806358337402344, 0.6587734222412109, -0.8614141941070557, -1.538083553314209, 0.42034274339675903, -0.0546543262898922, 0.8205752372741699, -0.8834806084632874, 0.228782057762146, -1.4690526723861694, 0.3730853199958801, 0.567945122718811, -0.34450963139533997, 2.355767250061035, -2.0113439559936523, 0.6120068430900574, -0.25057610869407654, 1.0528002977371216, 0.7125122547149658, 0.472030907869339, -0.7258562445640564, -0.5729790925979614, 2.0449769496917725, 0.038641806691884995, -1.97648286819458, 1.4439444541931152, 0.7249808311462402, 0.12057459354400635, -0.4574175775051117, 0.05830567702651024, 0.5266724228858948, -1.9848030805587769, 1.5237168073654175, 0.15649984776973724, -0.9812690615653992, -0.2573734223842621, -0.8139358162879944, 0.2284572273492813, 0.46254435181617737, -1.1201826333999634, 0.5830658674240112, -0.615756630897522, -0.29675278067588806, -1.0383691787719727, 1.3462152481079102, -1.3125489950180054, 0.7515881657600403, -0.3805862367153168, -0.4261859059333801, -0.8985738158226013, -0.20907093584537506, 0.23291811347007751, 0.15407590568065643, -0.0382143072783947, 0.27253979444503784, -0.07022630423307419, 0.17364093661308289, 0.0527946911752224, 0.48509180545806885, 1.6812984943389893, 0.5550679564476013, -0.2493956983089447, 0.17137981951236725, -0.2968932092189789, 2.3807358741760254, 0.678526759147644, 0.8534098863601685, 2.8487794399261475, -0.2508009374141693, -0.6243041753768921, 1.753207802772522, -1.0474278926849365, -0.8821297883987427, 1.8925234079360962, 0.40863093733787537, -0.3868284225463867, -1.8287328481674194, 0.9838855266571045, 0.02740224450826645, 1.5175011157989502, -0.12367930263280869, 0.9546438455581665, 0.8375266790390015, 1.2859292030334473, -0.057436373084783554, 0.28682610392570496, -0.05819018557667732, -0.18344606459140778, -0.987686276435852, -1.3159048557281494, 0.36841335892677307, -0.6329312920570374, 1.5377167463302612, -1.0911188125610352, -2.0961883068084717, -1.8125146627426147, 1.1716065406799316, 1.1099687814712524, 0.3463343381881714, -0.6464161276817322, 0.5306238532066345, -0.7939611077308655, 0.6404354572296143, -0.35306352376937866, 0.2862265706062317, -1.7843587398529053, 1.3188157081604004, -0.3186124265193939, 0.2519606351852417, -1.7994359731674194, 0.735927402973175, -1.578089952468872, -0.33419910073280334, 0.65623539686203, 0.9456786513328552, -0.9412510395050049, -1.1872478723526, 1.5749379396438599, -0.20122618973255157, -0.2393268197774887, 1.487095594406128, 0.10848765075206757, 0.10305823385715485, -0.32406148314476013, -0.42055821418762207, -0.5468884706497192, -1.5615681409835815, -0.29472124576568604, -0.3845173120498657, 0.15653160214424133, -0.571478009223938, -1.4883439540863037, 1.2172683477401733, -2.4762654304504395, 0.2994726896286011, 0.5824414491653442, 0.20543111860752106, 0.1298885941505432, -0.03941987082362175, -1.8229455947875977, -0.18537935614585876, -0.750101625919342, 0.0869014784693718, 0.3295295238494873, -2.7139809131622314, -0.14566314220428467, 0.18136204779148102, -0.036854568868875504, -1.0819000005722046, 1.6064122915267944, -0.342855304479599, 1.0414655208587646, -2.0328776836395264, -0.4744989573955536, 2.4997241497039795, 0.5296798348426819, 0.415723979473114, 0.7283223271369934, 0.05995801091194153, -1.9231746196746826, 1.2275100946426392, 0.10991612821817398, -0.11809271574020386, -0.8816317915916443, 0.5392275452613831, -0.8086276054382324, -0.5883493423461914, 0.07282087951898575, -0.6280902028083801, -0.9703258275985718, 1.608143925666809, -0.8749123215675354, -0.6022021770477295, 0.014289733953773975, -1.0714609622955322, 0.39543768763542175, -0.8647118806838989, 1.2061383724212646, -0.5424932837486267, -0.5178521275520325, 1.1522570848464966, -0.4655802249908447, -0.48470383882522583, 0.5009686946868896, 1.079758882522583, -0.5706230401992798, 2.369734525680542, -0.40295588970184326, -0.05721576139330864, -0.449117511510849, -0.8516720533370972, 0.36285537481307983, 1.2642192840576172, 0.43536800146102905, 1.0369274616241455, -1.165709137916565, -1.2170581817626953, -1.0038186311721802, -1.830417275428772, 2.825883388519287, -0.9827799201011658, -0.08111150562763214, 2.0478012561798096, 0.41025596857070923, -0.7939474582672119, 0.8999977707862854, 0.8097983598709106, 0.4135497212409973, 1.6828176975250244, -2.745213031768799, -0.7637192010879517, 1.0125490427017212, 1.473671317100525, 0.3024187386035919, -0.6895383596420288, -0.029166249558329582, 0.9402205348014832, -0.5914582014083862, -0.19840329885482788, 0.7423696517944336, -0.4410844147205353, -0.6109545230865479, -1.0920450687408447, 0.3168845772743225, 0.7705384492874146, 1.0747473239898682, 1.6549129486083984, -0.7544772624969482, 0.14381836354732513, -0.31817707419395447, 1.3373618125915527, 0.5743987560272217, -1.14699387550354, 0.27475354075431824, 0.3779104948043823, -0.7242801785469055, 1.1489958763122559, -1.2360600233078003, -2.1267127990722656, 0.4284661114215851, 0.2214503437280655, 1.0988510847091675, 3.21250057220459, 1.5922175645828247, -0.4853218197822571, 0.2847316861152649, 1.23738694190979, 0.700134813785553, -1.0839980840682983, 0.5131530165672302, -0.8879262208938599, 1.6268788576126099, -0.5534844994544983, 1.368097186088562], [-0.7404471635818481, -0.04709959402680397, 2.508739471435547, 0.5489809513092041, 1.3445966243743896, 0.629522442817688, -1.0597100257873535, 1.1904698610305786, 0.2957327663898468, 1.5615124702453613, 1.380304217338562, -0.4302368760108948, -0.13987308740615845, -1.2880780696868896, -0.48414668440818787, -0.666999101638794, 0.8447033762931824, 0.9223604202270508, 0.8524342179298401, 1.0404103994369507, -0.44134020805358887, -0.471135675907135, 0.06419117748737335, 0.9681370258331299, -0.27122703194618225, -0.5930310487747192, -0.17773692309856415, -1.1086608171463013, -0.5587825775146484, 0.5314393639564514, 0.6936131119728088, -0.7570893168449402, -0.310124009847641, 1.2847840785980225, 1.2391334772109985, -2.196718454360962, -0.6691610813140869, -0.513574481010437, 0.5116894245147705, -1.357375979423523, -0.3669620156288147, 0.6925743222236633, 1.0385034084320068, 0.33577224612236023, -0.8911088705062866, 0.19370420277118683, -0.6582793593406677, 0.21556732058525085, -1.056127905845642, 0.2478782683610916, -1.813940405845642, -0.9399557709693909, -0.8318939805030823, 0.7417340874671936, 1.6711002588272095, -1.4999275207519531, -0.2694394290447235, -1.3774142265319824, 2.191401720046997, -1.841167688369751, -0.3160219192504883, -1.5534248352050781, -0.3011375069618225, 0.13627348840236664, 0.9451563954353333, -0.7243047952651978, -1.1159484386444092, -0.18699347972869873, -1.424831748008728, 0.17511805891990662, 1.3106473684310913, 1.0765705108642578, -1.6715556383132935, -0.0018294367473572493, -1.1667536497116089, -1.2045260667800903, -0.5276793241500854, 0.38885748386383057, -0.6497316956520081, 0.6346634030342102, 0.20879840850830078, 0.32241228222846985, 0.2100227177143097, -0.23126552999019623, 0.8109362721443176, -0.8480560183525085, -0.5999597311019897, 0.21109405159950256, 1.318278431892395, -0.6023166179656982, -1.9268232583999634, -0.007126751355826855, -0.07283619046211243, -1.0582292079925537, 0.045422326773405075, 0.823159396648407, 0.9028199911117554, -1.4859758615493774, -0.7991797924041748, 1.8672668933868408, -1.7245908975601196, 0.41944971680641174, -1.1191926002502441, 0.867024302482605, -1.2616783380508423, -0.30300620198249817, 1.7859934568405151, 1.0409111976623535, 1.186008334159851, 1.189889907836914, 0.4826064705848694, 1.5884778499603271, -0.7642812132835388, -0.8852919340133667, 0.03646614030003548, 1.4855116605758667, 0.8918160796165466, 0.17209996283054352, -0.4689752757549286, -0.6641945838928223, -0.646697461605072, -1.0948052406311035, 0.39793166518211365, 0.7867905497550964, 0.3787177503108978, -0.43443045020103455, -0.7832041382789612, 0.424012154340744, -1.3402270078659058, -0.10957012325525284, -0.9920636415481567, 0.02538093365728855, -0.7397762537002563, 0.1657748520374298, 1.4575518369674683, 0.42132794857025146, -1.1751288175582886, -0.3867543339729309, 1.09311044216156, 0.41391873359680176, 0.3304595947265625, 0.012486463412642479, -0.2352486103773117, 0.5758805871009827, -0.41743820905685425, 0.39739853143692017, -0.6136475205421448, 0.7920017242431641, -0.99479079246521, 0.8710092902183533, -0.021680813282728195, -0.4708743691444397, -0.5356183052062988, -1.0194604396820068, 0.17412281036376953, 0.27916350960731506, -1.1271532773971558, -0.22105416655540466, 0.6319069266319275, -1.5065187215805054, 0.06828224658966064, -0.679638683795929, -0.026829395443201065, -0.10016867518424988, -0.06056183576583862, -0.2788427174091339, -0.8338112831115723, -0.7474107146263123, -0.48031461238861084, 0.6269446015357971, -0.3166831433773041, -1.856770396232605, 0.8890858292579651, 0.5966922640800476, -2.2402145862579346, -2.7494170665740967, -1.195298671722412, 1.019901156425476, 0.252311110496521, 0.2772492468357086, -1.1922454833984375, -0.22948651015758514, 0.3375098705291748, 0.8745015263557434, 0.5084078311920166, 0.782062292098999, 0.46764078736305237, -1.2227836847305298, 0.7010868787765503, 0.162925586104393, 0.49881860613822937, -0.6241328716278076, 0.8086727261543274, 1.7639498710632324, -0.2549106180667877, 0.6751663684844971, 0.214670330286026, -1.0392239093780518, 0.18212398886680603, 1.9451292753219604, 0.2791922986507416, -0.3794431686401367, -1.2410093545913696, 0.15492545068264008, -0.906723141670227, 1.874506950378418, -0.60654616355896, -1.1901955604553223, -1.1463987827301025, -0.6041579842567444, 0.15304847061634064, -0.018942417576909065, 2.6654937267303467, -0.03150348365306854, -0.5265094041824341, -0.15550574660301208, 1.6246635913848877, 1.42012619972229, -0.0200943760573864, 0.5638250112533569, 1.5465161800384521, 0.980574369430542, 0.30873969197273254, -0.264899343252182, -0.5381542444229126, -0.06311127543449402, -1.6460984945297241, 0.0034410886000841856, -0.1499626636505127, -2.251084566116333, -0.8531373739242554, -0.31387344002723694, -0.07506231963634491, -0.41381996870040894, 0.6053862571716309, 0.659339427947998, 0.4944552779197693, -1.8278961181640625, -0.3815443217754364, 2.389787197113037, -1.1960214376449585, -0.14030241966247559, 0.7112309336662292, 1.5995885133743286, 0.7850833535194397, -1.083876132965088, -0.07763217389583588, -0.8986395001411438, -1.0615075826644897, 1.2689402103424072, -0.9074901938438416, 0.4554242789745331, 2.555741310119629, 0.07882857322692871, 1.286529779434204, -1.50809645652771, -0.13164545595645905, -1.7748007774353027, -1.6539487838745117, -0.10524298995733261, 0.3135947287082672, -0.7578750252723694, -0.17806482315063477, 1.6382131576538086, 2.057779312133789, 0.20723474025726318, 1.359123706817627, -1.3965890407562256, -0.09544885903596878, 0.7105757594108582, 0.09409633278846741, 0.20817233622074127, 0.8908791542053223, 0.7534205317497253, 1.1346381902694702, 0.30883851647377014, -0.24148687720298767, -0.26200801134109497, 0.3777618706226349, 0.554544985294342, 0.6401224136352539, -1.1612541675567627, -0.7578542232513428, -1.61422860622406, -0.7861887812614441, 0.3233780562877655, 0.06969259679317474, -0.5163843631744385, 1.2695238590240479, 0.1549636274576187, -0.45796123147010803, 0.1404632180929184, -1.6322182416915894, -1.2857915163040161, 1.6438274383544922, 1.2510063648223877, -1.331664800643921, -0.46936875581741333, -1.5565897226333618, -1.979788899421692, -1.7470989227294922, -0.37817931175231934, 0.02743091993033886, -0.1790684312582016, 0.5065408945083618, 1.0806396007537842, -0.6278074979782104, 0.28016284108161926, 0.9802375435829163, -0.39899447560310364, -0.8718422055244446, 0.5664557218551636, 0.1039394661784172, -0.16190002858638763, -0.019850164651870728, -0.27576008439064026, 0.6660900712013245, -0.0992046371102333, -0.43043550848960876, -1.8919181823730469, -0.4456383287906647, -1.0347040891647339, -0.5687319040298462, -1.0346876382827759, -0.13842424750328064, 0.10692071914672852, 0.19120994210243225, -0.9422851204872131, 0.40637731552124023, 0.658149242401123, 0.1417955458164215, 0.42502206563949585, 0.8185849785804749, -0.17831851541996002, 0.05206311121582985, -0.575792133808136, -0.5334741473197937, 0.6561851501464844, -1.4796148538589478, -1.9109296798706055, -0.9048714637756348, 1.8999698162078857, -2.7371811866760254, 0.19825918972492218, -0.6966675519943237, -0.9745626449584961, 1.042039155960083, 1.6441986560821533, -1.628015398979187, 0.7668392658233643, 1.683455228805542, -0.4504051208496094, 0.11120335012674332, -1.2694039344787598, 0.08916317671537399, 0.11144422739744186, 0.6989635825157166, 1.567497968673706, -1.6513569355010986, 0.34594962000846863, -0.6167042255401611, 0.15419836342334747, 1.5239452123641968, 0.17690184712409973, 1.151600956916809, 0.37324395775794983, 0.27333882451057434, 0.5421300530433655, -0.22133225202560425, -2.1803510189056396, 1.1177027225494385, 0.4965617060661316, -1.1035414934158325, 1.6643779277801514, 0.671596348285675, 0.5713719725608826, 0.6758365035057068, 1.327839732170105, -0.9173680543899536, 0.6399677395820618, 0.924765408039093, 0.8782297372817993, 0.4269370138645172, 0.21539157629013062, -1.2406423091888428, 0.25315991044044495, -1.4829339981079102, 0.7876520752906799, 1.213857650756836, -1.2492073774337769, -0.3699401915073395, -0.6242303252220154, -1.6322638988494873, 0.8840456008911133, 0.7334092259407043, -2.631896495819092, -1.0548202991485596, -0.8618673086166382, -0.17657427489757538, 0.00043042789911851287, 1.1820157766342163, 1.0804948806762695, 0.025502748787403107, 0.9243026375770569, 1.315985083580017, -1.3107948303222656, 1.8847734928131104, 1.2056487798690796, 1.1983195543289185, 0.5735183954238892, 1.4066574573516846, 0.341063529253006, -0.5913227200508118, -1.6579355001449585, 1.0551869869232178, 0.2681201994419098, -0.6861488223075867, -1.8617786169052124, 1.359050989151001, -1.0449824333190918, 0.4306906759738922, 0.2222285121679306, -0.345210462808609, 1.6323052644729614, 1.0282150506973267, -0.6927729249000549, 1.4538065195083618, -0.038907431066036224, -1.195251226425171, -0.5450224876403809, -0.6323509216308594, -0.42507123947143555, -1.1158190965652466, -0.9973913431167603, -0.7814860939979553, 1.5438061952590942, -0.9413496255874634, 0.02195442095398903, 1.1230852603912354, -1.6926051378250122, 0.14279957115650177, -0.316718190908432, 0.016464723274111748, -0.5769568681716919, -1.0658963918685913, -0.46162259578704834, -1.576380968093872, 1.746335506439209, 0.9277231693267822, 0.07751751691102982, 0.4268947243690491, -2.245365858078003, -1.1368671655654907, -1.326054334640503, -0.7049674987792969, -1.3905448913574219, 1.1039024591445923, -0.9551893472671509, 1.2736890316009521, 0.675363302230835, 0.16010288894176483, -1.0290517807006836, -1.4044280052185059, 0.28438225388526917, 0.4457852244377136, 0.6512908935546875, 2.0372424125671387, -0.7511944770812988, -1.7018039226531982, -0.44861164689064026, 0.9198864698410034, -1.024957299232483, -0.539741039276123, -1.1580785512924194, 1.002998948097229, -1.9584987163543701, 1.1446983814239502, -2.0733845233917236, -0.10386567562818527, -0.2593909502029419, 0.10583086311817169, 1.8344234228134155, -0.18843834102153778, 0.2398497611284256, -0.19576692581176758, -1.5898411273956299, -0.3466341197490692, -0.3138086795806885, 2.120790481567383, 0.018266407772898674, 0.19724343717098236, -0.930878221988678, -0.16576306521892548, 0.4108601212501526, 0.049577973783016205, 0.5247805118560791, 0.2943519949913025, -1.2139031887054443, 1.254690170288086, -1.0612494945526123, -2.397214889526367, -0.12156988680362701, 1.60457181930542, 0.24642930924892426, 0.8875405788421631, 0.7055970430374146, -0.8773308396339417, 0.8548144698143005, 0.4111359417438507, -1.1316543817520142, -0.04101226478815079, 0.6109325289726257, -0.34169459342956543, -2.694596767425537, 0.6129027605056763, -1.2879770994186401, 0.3008401691913605, -0.22351549565792084, 1.4351727962493896, -0.36747613549232483, -0.5143952965736389, -0.10916321724653244, -0.4771828055381775, 0.39037731289863586, 3.3430137634277344, -2.127337694168091, 1.3887176513671875, -0.16295291483402252, 1.8099199533462524, 0.36490219831466675, 0.4377927780151367, -0.8316029906272888, 1.585568904876709, 1.096739649772644, -0.710574209690094, 0.0821184366941452, 0.006420640740543604, 2.1011064052581787, 1.0502078533172607, -0.2558479607105255, -0.8208691477775574, -0.9341118335723877, -2.4938125610351562, 0.5107410550117493, 1.1795169115066528, 0.29380708932876587, -0.7167273759841919, -0.14035774767398834, 1.2058274745941162, 0.35978493094444275, 0.3504446744918823, 0.6246665716171265, 0.799483597278595, 0.44643434882164, -0.5725381374359131, -0.16641978919506073, -0.0926787331700325, 0.3128105700016022, -2.3368184566497803, 1.4978400468826294, -0.5477883815765381, 0.10450471192598343, 0.6332005858421326, 1.0792123079299927, -0.21996431052684784, -1.0693658590316772, -0.27541109919548035, -0.8645033836364746, 1.2884953022003174, 0.8911192417144775, -0.7830225825309753, -0.9522606134414673, 0.5178967714309692, -1.743765950202942, 0.5889143943786621, 0.11527286469936371, -0.28448694944381714, 0.14140789210796356, -2.0638437271118164, -1.2111256122589111, -0.7829185724258423, -1.9136930704116821, 1.030035376548767, 0.7761386632919312, -1.5396357774734497, 0.7575183510780334, 0.10382407903671265, 1.0861070156097412, 1.0111812353134155, 0.2814350128173828, -0.34715908765792847, 0.5656057596206665, -0.9673298597335815, -1.267604112625122, 1.5931147336959839, -1.2093368768692017, -1.411902666091919, 0.9113293290138245, 1.1156738996505737, -0.5592414140701294, 1.6862916946411133, -0.10991410911083221, -1.0683614015579224, 0.839733898639679, 0.9117127060890198, -1.7660115957260132, 2.0176756381988525, -1.1153062582015991, 2.057478189468384, -0.5148463249206543, -1.1109548807144165, -2.2697665691375732, -2.4749937057495117, 0.2083151489496231, 0.7230439782142639, 0.766392707824707, -0.02745521068572998, 0.32430270314216614, -1.1958329677581787, -0.9514473676681519, 0.41645780205726624, 0.06421725451946259, -0.4248324930667877, -0.2919263243675232, -1.1763620376586914, 0.7708914875984192, -1.057474970817566, 0.4317392408847809, -1.9198222160339355, 0.8210498094558716, -1.84714674949646, -0.010060475207865238, -1.240106463432312, -0.4010095000267029, 0.8517575263977051, -1.1731071472167969, -0.5738566517829895, -0.09480106085538864, 0.3071173429489136, -0.9051147103309631, -1.073410153388977, -0.702975332736969, 0.12214778363704681, 0.3543568551540375, -0.0671505555510521, -0.9579401612281799, 0.9186761975288391, -0.23386014997959137, 1.153881549835205, 0.6185839176177979, -0.3202318847179413, -1.4022393226623535, 0.6527911424636841, -0.2382669597864151, -0.3470280170440674, 0.15329939126968384, -0.6952222585678101, 0.25358688831329346, 1.032867193222046, 1.6714895963668823, 0.2935512065887451, -0.7125174403190613, 1.84268057346344, 0.6616013646125793, -0.40597614645957947, 0.12518785893917084, -0.07014377415180206, -1.7531116008758545, -0.5477052330970764, -2.1723310947418213, 0.5814859867095947, -1.0087571144104004, 0.41307711601257324, -0.05394696444272995, 0.8367385864257812, 0.18331578373908997, -0.8148342967033386, 1.1265209913253784, 1.0509377717971802, 0.514255702495575, -1.024591088294983, -0.708628237247467, 1.1446796655654907, -0.22870071232318878, -0.9019249081611633, -0.15983666479587555, -0.5894191265106201, -0.09947402030229568, -1.4351204633712769, -1.506123423576355, -0.09652383625507355, -0.5266608595848083, -0.31260132789611816, 0.6720105409622192, 0.7181487679481506, 0.9012734293937683, 0.1656492054462433, 0.4207555055618286, -0.36512333154678345, 1.4365910291671753, 1.1414467096328735, 0.5808301568031311, -0.21532012522220612, -0.32111725211143494, 0.025902893394231796, 0.25070738792419434, -1.8220460414886475, 1.1914652585983276, -0.10234750807285309, 2.4221808910369873, 0.1350308209657669, 0.6294946074485779, -0.6683893203735352, -1.368808388710022, 0.8480854630470276, 1.3444294929504395, -0.41776666045188904, 1.1250522136688232, 2.4664816856384277, -1.2141940593719482, -0.25943857431411743, 0.4650658071041107, -0.04546982795000076, -2.1184825897216797, -0.38934919238090515, -1.3848341703414917, -0.7904824614524841, -0.03478974103927612, -1.1929205656051636, -0.6595544219017029, -1.185716152191162, -0.5713546872138977, -0.8191872239112854, 0.3369503915309906, -0.153328537940979, 0.9820985198020935, -0.5343499779701233, -0.12600012123584747, -2.5278074741363525, -0.8527483344078064, 0.3580617904663086, 0.4243305027484894, -1.2570651769638062, 0.4861094653606415, 0.05157497525215149, 1.0610564947128296, 0.016803154721856117, 0.0036586755886673927, -0.35614022612571716, 0.5158717632293701, 0.782278299331665, 0.9543682336807251, -0.1605158895254135, -2.3206019401550293, -0.3022049367427826, 1.2780511379241943, 0.09954096376895905, -0.9488813877105713, 0.7545840740203857, -1.061928391456604, 2.3946731090545654, 0.5748383402824402, 1.2341017723083496, -0.12716683745384216, -2.644580364227295, 1.3245521783828735, 1.8326220512390137, 0.08432675153017044, -0.6044259071350098, 0.8107331991195679, 0.9520715475082397, -0.9902324676513672, -0.12792488932609558, -1.2772107124328613, -0.7746437191963196, 2.208346366882324, -0.1918894499540329, 0.9062093496322632, -0.17215941846370697, -1.364446997642517, 0.34047046303749084, 0.1298341155052185, -0.6597086191177368, 0.745510995388031, 0.3759128153324127, -2.0306708812713623, -0.9528733491897583, 1.1448544263839722, 0.8725045323371887, 0.5260125398635864, -0.9515962600708008, 2.0805001258850098, -0.269111692905426, 0.5602781176567078, -1.0197902917861938, -0.7687193155288696, 0.26316407322883606, 1.3943097591400146, 2.0531554222106934, 0.4726831614971161, -1.902613878250122, 0.029985200613737106, -0.7733132243156433, -2.093385696411133, 0.36183276772499084, 0.0026139707770198584, -0.1140662357211113, 1.3176124095916748, -0.6782810091972351, 0.45511260628700256, -1.2311630249023438, -1.8138386011123657, -0.08014178276062012, 1.3484463691711426, -0.8989239931106567, -0.7331740260124207, 0.9237858057022095, -0.6650424003601074, -0.5246900916099548, -0.6107749342918396, 1.0986484289169312, -0.2415139079093933, -1.0379756689071655, -1.5460495948791504, 1.527184247970581, -0.6930730938911438, -0.28338226675987244, 2.463811159133911, -0.39423710107803345, 0.34550997614860535, 1.9941009283065796, -1.3916847705841064, -0.4619792401790619, 1.457533836364746, 0.46524983644485474, -0.1998659372329712, -0.10453824698925018, -0.5626804828643799, -0.41867348551750183, -1.1590863466262817, -1.1315654516220093, -0.9537040591239929, -0.13060662150382996, -1.8546900749206543, -0.513391375541687, 1.2765072584152222, -1.3885767459869385, -0.032565902918577194, -1.399702787399292, 1.188768744468689, -1.4558537006378174, 1.1278152465820312, -0.5931474566459656, 0.666890025138855, 0.3489309251308441, 0.19267524778842926, -1.4161947965621948, 0.06233823671936989, 2.3316032886505127, -0.4014265835285187, -1.3516755104064941, -0.3985060751438141, -0.14680376648902893, -0.016298910602927208, -1.3382725715637207, -2.2714345455169678, -0.2630932927131653, -1.0840039253234863, 1.4675078392028809, 1.5094319581985474, 1.0278087854385376, -0.6642730236053467, -2.49299955368042, -0.9045978784561157, 0.3556276261806488, -0.8760371208190918, 0.530754029750824, 1.4493238925933838, 0.1086912453174591, -0.9095159769058228, 1.603690266609192, -0.6127105951309204, 0.17033444344997406, 0.00998745858669281, -0.8979159593582153, 0.9886300563812256, 1.1165275573730469, 0.8239602446556091, 1.0154606103897095, -1.261443018913269, -0.04400966316461563, 1.77151620388031, -0.8038263916969299, 0.4708573818206787, 0.8438758850097656, -0.11767327040433884, -0.6969442963600159, -0.08515628427267075, -0.8838406801223755, 0.3357102870941162, -0.7476683259010315, 0.07847841829061508, 0.7740291357040405, -0.5229119658470154, 1.6671855449676514, -1.6672723293304443, 1.427491545677185, 1.1835678815841675, -0.9857539534568787, 0.6150205135345459, -0.7221278548240662, -0.8408892154693604, -1.0947530269622803, -0.8908363580703735, 0.34362533688545227, -0.5251225233078003, -0.14052355289459229, -1.0723553895950317, -0.2340436428785324, -1.4407902956008911, -2.0680549144744873, 0.014644529670476913, 0.08736204355955124, 0.8127212524414062, -1.3858509063720703, -0.04781598597764969, -0.3377421200275421, -0.5214475393295288, -1.1137328147888184, 1.6110316514968872, -0.8324018120765686, 0.23914112150669098, -0.9237878322601318, -2.125149965286255, -0.984554648399353, 0.9315844774246216, 1.057605504989624, -0.04072704166173935, 0.7704911828041077, 0.26197993755340576, -0.6667611002922058, -1.7930290699005127, 0.4643992483615875, 0.8176669478416443, 1.0296387672424316, -0.2941453158855438, -0.9463694095611572, 0.14114241302013397, -0.9415478706359863, 0.9321287870407104, -0.7347524166107178, 0.9311012625694275, -0.19104722142219543, -0.09124220907688141, 0.25711527466773987, -0.649583101272583, 0.02184971608221531, 0.45970308780670166, -0.07981615513563156, -0.9026134014129639, 2.3105084896087646, 0.5005201101303101, -0.33225545287132263, -0.057706110179424286, -3.2643065452575684, 1.2257274389266968, -0.4528714120388031, 0.02509389817714691, 0.18237023055553436, 0.04924442991614342, -1.1550686359405518, 1.6458501815795898, 0.11814337223768234, -0.5154825448989868, 0.04847821220755577, 0.301486611366272, -1.3889070749282837, 0.23225364089012146, -1.4064475297927856, -0.3173624873161316, 2.5250885486602783, 0.6548595428466797, 1.5832089185714722, 0.2397724688053131, -0.38002532720565796, 1.2070642709732056, 1.120922327041626, -0.060076162219047546, -0.5477113723754883, -0.17076246440410614, -0.6616901755332947, 0.9868680834770203, 1.620442271232605, -0.026427527889609337, 0.149887353181839, -0.4147571325302124, -1.5809814929962158, -0.0252195093780756, 0.9833971858024597, -0.9547242522239685, -0.11889685690402985, -0.8793588280677795, 1.3544071912765503, -0.024645457044243813, 0.011066862381994724, 0.12559983134269714, 0.5807046294212341, 0.27786704897880554, -1.4176373481750488, 0.8407386541366577, -0.09141481667757034, -0.05963898077607155, 2.063413381576538, 0.6122553944587708, 0.8522158861160278, 0.6550664901733398, -0.08681967109441757, 0.015404835343360901, 1.0813789367675781, -1.2275794744491577, -0.44733089208602905, 2.403425693511963, 1.5507030487060547, -0.9831575155258179], [1.1370102167129517, 1.8958954811096191, -0.6601591110229492, -0.8250978589057922, -0.3884999752044678, 0.18087437748908997, 0.6599663496017456, 0.0151260020211339, 0.5914226770401001, 0.09752589464187622, 1.7874377965927124, -0.4361572563648224, 0.545122504234314, -1.4835582971572876, 0.20601136982440948, -1.682364583015442, -0.052082158625125885, -2.0476479530334473, -0.7761955857276917, 0.03234695643186569, -0.2689538300037384, -0.11547999829053879, 1.7940731048583984, 1.2248306274414062, -0.07950117439031601, 0.0908438190817833, -0.5362986326217651, 0.7561932802200317, -0.6550758481025696, -0.6655105352401733, -0.5706741213798523, -2.6107027530670166, -0.3192262649536133, -0.10874508321285248, 0.4638192057609558, -0.5191471576690674, -0.5718362331390381, 1.2221673727035522, 0.7848063111305237, 0.21038521826267242, -0.38503748178482056, 0.2789904475212097, -0.19331172108650208, -0.11002825945615768, -0.7466503977775574, 0.21163567900657654, -0.6307227611541748, -0.40976482629776, -0.4396911859512329, 0.15047378838062286, -1.0425281524658203, 0.04311847314238548, 0.7188553214073181, 0.4083907902240753, 1.0516581535339355, 0.532192051410675, 0.7276766300201416, 1.7094956636428833, 0.530706524848938, -0.33522483706474304, -0.18164126574993134, -0.28581780195236206, 0.23493215441703796, 0.28532519936561584, -1.5625883340835571, -1.5936099290847778, -1.9098165035247803, -0.018404681235551834, -0.1241644024848938, -0.1827191859483719, 0.6597368717193604, 0.27437272667884827, 0.21592244505882263, 1.134461760520935, 1.6148793697357178, 0.6897392868995667, -0.9073375463485718, 1.012721061706543, -1.5099117755889893, -0.030678480863571167, -0.7963860630989075, -0.9810193181037903, 1.9303443431854248, 0.6525881290435791, 0.25311487913131714, -0.8968249559402466, -0.9185715317726135, -0.20647750794887543, -0.08009223639965057, 0.6729148626327515, 0.4103843569755554, 0.5548884272575378, 0.5370997190475464, -0.1689991056919098, -0.3508015275001526, 1.020430088043213, 0.8776687383651733, 0.261991024017334, -0.5450754761695862, 0.38559192419052124, 0.2566872239112854, 0.17110493779182434, -0.8608098030090332, -1.4081517457962036, -1.1243561506271362, 0.013651866465806961, -0.5588815212249756, 0.6423969864845276, 0.09677860140800476, 0.9030603170394897, -0.5128431916236877, -1.1735484600067139, 3.0522682666778564, 0.27806100249290466, 1.6527711153030396, -1.8687098026275635, 0.31798699498176575, -0.7986823320388794, 0.040418099611997604, -0.9459347128868103, -2.149543523788452, -1.1202349662780762, 1.322645902633667, 1.3284668922424316, -0.7156693935394287, -0.05998431146144867, 0.548999547958374, -0.1970420479774475, -0.589648425579071, -0.6000640988349915, 0.23994742333889008, -0.7713674306869507, -1.9599504470825195, 1.353672742843628, 0.7014885544776917, -0.5936964750289917, -0.5134497284889221, 1.7398101091384888, -0.4416177272796631, -0.1426618993282318, -2.096122980117798, 0.6912844777107239, -0.5270752906799316, 0.9347606897354126, 1.1979881525039673, -0.43695366382598877, 0.21363644301891327, 2.0967867374420166, -0.47088485956192017, 0.8226845264434814, 0.6022773385047913, -0.7715505957603455, 0.46985793113708496, 1.0425764322280884, 1.210699200630188, -0.8154128193855286, -0.6404054164886475, -0.09696771204471588, 1.0920090675354004, -1.2253327369689941, -0.770061731338501, -2.115128517150879, 1.789703607559204, 0.6416509747505188, 1.0375707149505615, -0.6271867752075195, -0.9961287379264832, 1.4633947610855103, -0.589658260345459, -0.14649513363838196, -0.700375497341156, 0.044941820204257965, 1.9472391605377197, 0.9447993636131287, -0.8570852875709534, -0.014042367227375507, 1.2852277755737305, 0.2201881855726242, 0.23717284202575684, -0.7999274730682373, -0.6366257071495056, 1.0464632511138916, -0.6223945617675781, -0.8009300231933594, 1.9548039436340332, 0.5242244601249695, -0.13487914204597473, 1.0377774238586426, -0.9566175937652588, 0.3442835807800293, -0.6712679266929626, 1.272608995437622, 2.1648614406585693, 0.38888925313949585, 2.1991028785705566, 0.6520954370498657, -1.3477847576141357, 0.0003791575727518648, 1.609176754951477, -1.339566707611084, 1.3578457832336426, -0.9070573449134827, -0.22361326217651367, -0.13212792575359344, -1.2806189060211182, -1.1906239986419678, 0.0638638287782669, -0.6443385481834412, -0.6873091459274292, 0.7949104905128479, 0.550671398639679, -0.021239180117845535, 1.405413031578064, -0.3988710939884186, 1.4030439853668213, -1.290124773979187, 0.05700448527932167, -0.6025747060775757, -0.47215089201927185, 0.536798894405365, 0.5579968094825745, 0.3644324243068695, -0.8814616203308105, -1.4693814516067505, 0.9472525119781494, 0.5068972706794739, 0.8828917145729065, 0.18090961873531342, 1.05254065990448, -0.22238798439502716, 0.6256490349769592, -0.24913987517356873, 1.9812182188034058, 0.25291064381599426, 0.026248501613736153, -0.022814590483903885, 0.11239679902791977, -1.0360366106033325, 0.7263676524162292, 1.5317457914352417, -1.15437650680542, -0.5158991813659668, 0.9569883942604065, 0.8637263774871826, 0.24749699234962463, 0.8017746210098267, -0.3352588713169098, -0.32828500866889954, 0.9799670577049255, -0.239426389336586, -1.0957905054092407, 1.7778962850570679, 1.6762081384658813, 0.11566976457834244, 0.052279386669397354, 0.9918075203895569, -0.6743733286857605, 0.8704227209091187, -0.13557031750679016, 1.582882285118103, -1.1305474042892456, -1.644357681274414, -0.18629604578018188, 1.1070058345794678, -0.8690148591995239, -0.31012141704559326, -0.2614874839782715, 0.11174452304840088, 0.8858100175857544, -1.2859073877334595, -0.5295476317405701, -1.2503881454467773, 0.04341062530875206, -1.1900357007980347, -0.007442839909344912, -0.13805297017097473, 0.7208049297332764, 1.2365082502365112, -0.3665342330932617, -0.8119491338729858, 0.1402405947446823, 0.7166231870651245, -0.979800283908844, 0.6410670876502991, 0.3875386714935303, -1.079509973526001, -1.4233404397964478, 0.8743321895599365, -1.3996585607528687, 0.8906716108322144, -0.32309862971305847, 0.09302601218223572, 0.09387283772230148, -1.2395628690719604, -1.3857306241989136, 0.5736700296401978, 0.7985774278640747, -0.7435017228126526, -1.050577998161316, 0.3326171636581421, -0.5776063799858093, -1.3801281452178955, -0.19491954147815704, -1.4580327272415161, 0.45742449164390564, -0.28606468439102173, 0.05964642018079758, -1.4887182712554932, 1.7185946702957153, 0.7716074585914612, 1.6772993803024292, 0.8085602521896362, -0.9171757102012634, -1.0802732706069946, -0.2891429662704468, -1.2431849241256714, 0.5759851336479187, -0.2914394438266754, -0.09930713474750519, -0.3159249424934387, 0.7285054326057434, -2.771700859069824, -0.4113626778125763, -0.16744142770767212, 0.5446829199790955, -1.0089735984802246, -0.5814668536186218, 0.4619014263153076, 0.7249252796173096, 1.9188854694366455, -0.5084814429283142, 0.1723007708787918, 1.5716708898544312, 0.17376023530960083, -0.018112363293766975, 1.4384924173355103, 1.3496888875961304, 1.9459643363952637, 0.13814343512058258, 0.5673415660858154, 1.1742734909057617, 1.1208062171936035, -1.581259846687317, 0.183048278093338, -0.15711279213428497, 0.2355596274137497, 0.16298693418502808, -0.16297778487205505, -0.35738131403923035, 2.70680832862854, -0.49302157759666443, 0.049134980887174606, 0.4675697684288025, 0.7410213351249695, 0.6806471943855286, -1.1841132640838623, -0.8335608243942261, -0.37373417615890503, -0.29022037982940674, -0.7667986154556274, 2.0114569664001465, 1.9791340827941895, 0.17519117891788483, 1.7166926860809326, 2.30220365524292, 1.9596965312957764, 0.9071674346923828, -0.7055094838142395, -0.29676875472068787, 0.011268827132880688, -1.735163927078247, 0.2643130123615265, -0.29892274737358093, -0.6239440441131592, 0.3554135262966156, -0.024297218769788742, -0.6799825429916382, -1.1612199544906616, -0.46502819657325745, 0.029823577031493187, 0.7775169610977173, -0.7466051578521729, -0.6769002676010132, -0.41469553112983704, 0.9051216840744019, -1.2700695991516113, -2.3117616176605225, 0.5063098073005676, -0.014218430034816265, -2.2284417152404785, -1.2943902015686035, 1.8281159400939941, 0.6252360939979553, -0.2648032009601593, 1.551145076751709, 0.574598491191864, 0.5279402136802673, 1.750737190246582, 0.8295543789863586, 1.6952053308486938, -1.3265613317489624, -0.07876131683588028, 2.0395238399505615, -0.31147468090057373, -1.05457603931427, 1.1049555540084839, 0.02820023149251938, -2.1108362674713135, 0.5054972171783447, -1.3312369585037231, -0.5160847306251526, -1.3920025825500488, -0.5933979153633118, -0.4264461398124695, -1.16167151927948, -0.7049912810325623, -0.6224870681762695, 0.00684774573892355, 0.19455161690711975, 0.21887758374214172, 0.008464102633297443, -0.9619150757789612, 0.6673328280448914, 0.669853925704956, -0.16015450656414032, 0.03240149840712547, -1.1805419921875, -0.20071107149124146, 0.45252248644828796, -2.96014666557312, 0.7000071406364441, 0.13821667432785034, 0.5610445737838745, 1.4922022819519043, 2.4383013248443604, -1.8634907007217407, -0.532504677772522, -0.057990867644548416, 0.8632898330688477, -0.04096095636487007, 0.854110598564148, 0.7742447257041931, 0.41521719098091125, 0.22632308304309845, -0.03847009688615799, -0.47687074542045593, -1.2337983846664429, 0.1135103702545166, 0.21359188854694366, -0.17180563509464264, 2.4926013946533203, 0.24625007808208466, 0.19197623431682587, 0.6261149048805237, -1.1811745166778564, 1.3706700801849365, -0.14619675278663635, 0.08144113421440125, -0.9947134852409363, 1.0294110774993896, 0.29788973927497864, 0.46288132667541504, -1.4768919944763184, -0.0923733338713646, -2.3273186683654785, 0.12437199056148529, 0.051894981414079666, 1.1429741382598877, -0.5532228350639343, 0.3961963355541229, -0.08612730354070663, 1.4754644632339478, 0.8502253293991089, -1.2899343967437744, -0.693609356880188, -0.18116538226604462, -1.1722818613052368, -0.050745997577905655, 1.5274800062179565, 0.27633431553840637, -0.9389654397964478, 0.5557039380073547, -0.06422989070415497, 0.34482449293136597, 0.5449655055999756, 2.050865888595581, 0.24736832082271576, -1.5809962749481201, -0.239529550075531, 0.2612055838108063, -0.9320442080497742, -0.8459045886993408, 0.3200512230396271, 0.8760760426521301, 1.1542432308197021, -0.17724056541919708, 0.18103766441345215, -0.28270432353019714, -0.39322587847709656, 0.678211510181427, 0.1602034568786621, -0.534638524055481, 0.94139164686203, -0.37146276235580444, 0.6448190212249756, 0.1302199512720108, -1.8453662395477295, -1.2371442317962646, 0.13847342133522034, -0.2844218313694, 1.119943618774414, 1.5456407070159912, 0.19060172140598297, 0.15300743281841278, 0.15787269175052643, 0.5264315009117126, 1.0788723230361938, 1.7532192468643188, -0.6646941900253296, -0.6902134418487549, -2.105682849884033, 0.5468229651451111, 0.9699275493621826, -1.868039608001709, -0.4687865674495697, -0.03546901419758797, -1.886799693107605, 0.38316962122917175, -1.1859933137893677, -0.305023193359375, 1.4450603723526, -1.3003629446029663, 0.27701082825660706, -0.032525885850191116, 0.8114215731620789, -2.303649663925171, -1.0840799808502197, -0.14617083966732025, 0.5855772495269775, 0.8352615833282471, -0.7293133735656738, -0.6065337657928467, -0.2543815076351166, -1.4349068403244019, -1.5529481172561646, -0.38733261823654175, 0.9618062973022461, 0.5338468551635742, -1.3281798362731934, -0.28374651074409485, -0.22288471460342407, -1.3376713991165161, 0.4749006927013397, 0.5891494750976562, 0.5362161993980408, -0.951250433921814, -0.2309977263212204, -0.9341105222702026, 0.25563839077949524, 0.7507134079933167, -0.8641009330749512, -0.7396380305290222, -0.017438894137740135, 0.7460476160049438, -2.0804381370544434, -0.009239256381988525, 0.8995441794395447, -0.10650187730789185, 0.8512441515922546, 0.21289417147636414, -1.7764744758605957, 1.6068764925003052, 0.6915651559829712, -0.6913660764694214, -0.8625555038452148, -1.3263177871704102, 1.1590328216552734, -0.34094008803367615, -0.6715236902236938, 1.3934133052825928, 2.0409562587738037, 0.6471679210662842, -0.2953381836414337, 0.29390162229537964, 1.0427825450897217, -1.0549495220184326, -0.26805785298347473, -2.3644919395446777, -1.748452067375183, -1.3062472343444824, -0.9021233320236206, -0.20695841312408447, -0.23875431716442108, -0.48148638010025024, -0.11563889682292938, -1.1482070684432983, 1.1265252828598022, 0.12501630187034607, 1.0422743558883667, -2.2244930267333984, 0.5766870975494385, 0.7025189995765686, 0.7168616056442261, -1.2657335996627808, -0.6822043657302856, 1.3562949895858765, 0.07229875028133392, -0.14289604127407074, 1.418247938156128, 0.4433939754962921, -0.9742031097412109, -0.9491742253303528, 0.76341712474823, 1.038095474243164, -1.280426025390625, -1.3929897546768188, -0.17320750653743744, -0.14526019990444183, 0.548445463180542, -0.7068953514099121, 1.003825306892395, -0.04767502844333649, -0.8831043243408203, 0.9458367824554443, 0.37002891302108765, -1.0759836435317993, -1.0103535652160645, -0.4080944061279297, 0.6288509368896484, -1.560926914215088, -0.7305534482002258, -0.004727588035166264, 1.1033246517181396, 0.16411200165748596, -0.5936049222946167, 1.4035371541976929, -0.23313413560390472, 1.7016608715057373, 0.19963358342647552, 0.6549421548843384, -0.6379327774047852, 2.314293384552002, 2.400947093963623, -1.8146520853042603, 0.9244702458381653, 1.1522718667984009, -0.12517035007476807, 0.4421098232269287, -0.06942626088857651, 0.11516347527503967, 0.4225110411643982, -0.9265965819358826, 0.24217402935028076, -1.6584298610687256, 2.5265421867370605, 0.10337086766958237, -0.9445856809616089, 0.4059401750564575, -1.0693459510803223, -1.4987066984176636, 0.7046982645988464, -1.6903313398361206, 0.4196341335773468, -0.6378006935119629, 0.08870397508144379, 0.9425598382949829, -1.7030962705612183, -1.4320287704467773, -1.007190227508545, 0.9425815343856812, 2.152688503265381, 0.2938973307609558, 0.10745401680469513, 1.2658246755599976, -0.5704644322395325, -2.2840352058410645, -0.5206709504127502, -0.6442863941192627, -1.505387783050537, 1.0660911798477173, 0.1837967038154602, -0.4325249493122101, -0.896281361579895, -0.9874885678291321, -1.841616153717041, -1.0842067003250122, -0.7133049964904785, 1.564710021018982, -1.4096860885620117, 0.5100733041763306, 0.19286563992500305, 1.043269395828247, -0.22093133628368378, -0.04259805008769035, 0.487754762172699, 1.283421516418457, 0.9405570030212402, 1.548683524131775, -0.6920788288116455, -0.18999549746513367, 0.05499352142214775, -1.0495762825012207, -0.2285412698984146, -1.388299822807312, 0.6372025012969971, -0.9440969824790955, -0.787926197052002, 0.9465104937553406, 0.8573867678642273, -1.5959923267364502, 0.13939061760902405, -0.8738030791282654, -0.9930835366249084, 1.9976803064346313, 1.0997231006622314, 0.4668814539909363, 1.3492093086242676, 0.39346230030059814, 0.6314168572425842, 0.7803412079811096, 0.9008299112319946, -0.5624983310699463, -0.18639671802520752, -1.1801060438156128, 1.1915581226348877, 0.09877010434865952, 0.6013577580451965, 0.31054723262786865, -0.5925590395927429, 0.06885267049074173, -0.5842204689979553, -0.5755655765533447, -0.0014506973093375564, 0.765977680683136, 1.4614120721817017, -0.30386853218078613, 1.4575363397598267, 0.15088990330696106, 0.8624913096427917, -1.451505422592163, 0.4048554301261902, -0.13954387605190277, -0.19054552912712097, -1.3157382011413574, 0.5001294016838074, 1.2318435907363892, 0.24990837275981903, 0.22769787907600403, -0.4889921247959137, -2.4725260734558105, 0.04505364969372749, 1.9828988313674927, -1.4909104108810425, 1.2023272514343262, 0.30260977149009705, -0.5991673469543457, -1.5424163341522217, 0.5476022958755493, -0.1871185451745987, -1.1995279788970947, -0.20056167244911194, 0.7766014933586121, -0.7528678178787231, -2.4220306873321533, -1.100114345550537, -0.522347629070282, 1.8676793575286865, -0.2061813622713089, -0.14786337316036224, 1.9997568130493164, 1.4228277206420898, -0.767586886882782, 0.5502067804336548, -1.222379446029663, 0.23061750829219818, 1.4533541202545166, 0.7425448298454285, 0.27163124084472656, 1.041261911392212, 0.46261805295944214, 0.7189120054244995, 0.6826465129852295, -0.7274238467216492, 0.8373949527740479, 0.7944048047065735, -0.5760459899902344, 0.6151123642921448, -1.0296330451965332, 0.542670726776123, -0.5001469254493713, -0.3675653636455536, 0.24417981505393982, -1.1150164604187012, -0.4722374677658081, 1.0854872465133667, 0.6300160884857178, 0.3819008469581604, -1.4531328678131104, -1.1124398708343506, 0.49325940012931824, -0.1929665505886078, -1.1833879947662354, -2.9438138008117676, -0.6072408556938171, -0.3167112171649933, 0.32786867022514343, -0.48237669467926025, -0.3434585630893707, -0.7321295738220215, 0.34648144245147705, -0.8599464297294617, 0.3309381306171417, 0.4794772267341614, 0.10783173143863678, 1.0697239637374878, -0.31702420115470886, 1.620162010192871, -1.3812822103500366, 0.8837406635284424, -0.1788175106048584, -1.0525016784667969, 1.4496798515319824, -1.2359288930892944, 1.4758877754211426, 0.04146099090576172, 1.3611253499984741, 1.3693082332611084, 1.3677740097045898, 0.8967509865760803, 0.06198633089661598, 0.3214927911758423, -0.9378381371498108, 0.12984682619571686, 0.3156251013278961, -0.8925900459289551, 2.0739920139312744, 2.7483696937561035, -0.16717872023582458, 0.1294504702091217, -0.5605849623680115, 1.314609408378601, 0.7758907675743103, -0.10673701763153076, -0.13798528909683228, 0.3694455921649933, 1.1493738889694214, 0.1768297404050827, 0.11882846057415009, -2.0226783752441406, 2.222195863723755, 1.0316951274871826, -1.2487291097640991, -1.4133192300796509, -0.17378391325473785, -0.08602907508611679, 2.1474084854125977, -1.6370211839675903, -0.24454818665981293, 1.7042546272277832, 0.2572353184223175, 0.811019778251648, -1.1315828561782837, -0.6536060571670532, -0.22342082858085632, -1.6545854806900024, 0.6019704341888428, -0.8619993329048157, 1.567624568939209, -1.639316201210022, -0.09626006335020065, 0.06551960855722427, 1.4885690212249756, -2.631251096725464, 0.3963744640350342, 0.33192259073257446, 2.143277883529663, -2.123903274536133, 1.4754196405410767, -0.46982815861701965, 0.07003746181726456, -0.11494359374046326, -1.6195099353790283, 0.5713458061218262, 1.2114629745483398, -0.6352011561393738, -1.2067110538482666, -0.9833353161811829, -0.3423410654067993, 0.892382025718689, 1.1758204698562622, -1.206610918045044, 0.07634235173463821, 0.2453511655330658, 0.955910861492157, -0.06441149860620499, -0.03187316283583641, -0.46897581219673157, 0.19650152325630188, 0.23368394374847412, -1.3870738744735718, -0.875096321105957, 0.4171452224254608, -0.5397809147834778, -0.3371714651584625, -0.9186530709266663, -0.6165603995323181, -1.0456582307815552, -0.00542715098708868, -0.17838208377361298, 1.7200552225112915, -0.6247789263725281, 0.6112499833106995, -1.8760133981704712, -0.5006612539291382, 0.8736254572868347, 0.28417009115219116, 0.9677901864051819, -0.20131142437458038, 0.15973971784114838, 0.2257249802350998, 0.8616387844085693, -0.21121320128440857, -2.565030813217163, -2.6109988689422607, -1.3055709600448608, 0.5750405192375183, -0.9874290823936462, -2.6366117000579834, 0.9171229004859924, 0.1573028713464737, -1.2463669776916504, -0.9276465773582458, 1.0811591148376465, -0.9775886535644531, -0.4100780785083771, 0.7216350436210632, 0.7581624984741211, 1.4013704061508179, -0.3644915819168091, -0.6320542097091675, -0.9623421430587769, 1.0882604122161865, -1.4609390497207642, 0.43544143438339233, -0.5089595317840576, 0.06952597200870514, -0.9031569361686707, 2.3360793590545654, 0.9321177005767822, -0.3359808027744293, 2.1183276176452637, 0.9573398232460022, -0.3698846101760864, 0.8885560035705566, 0.4333450496196747, 0.34314417839050293, 0.8881639838218689, 0.3222105801105499, -0.9143044352531433, 0.3359764814376831, 0.496054470539093, -2.098332166671753, 0.3670186400413513, -0.10885483026504517, -1.0360064506530762, -0.5186269283294678, 0.06327207386493683, -2.0475924015045166, -1.01613450050354, 0.6192368268966675, 0.5058500170707703, -1.0879689455032349, -0.45041775703430176, 1.1548131704330444, -2.9076457023620605, -0.7849193215370178, 0.8413955569267273, -2.4403607845306396, -0.9088093042373657, -1.10170578956604, -0.29555973410606384, 0.35415714979171753, -1.091598391532898, 1.209187626838684, -0.028423747047781944, 1.9345895051956177, 1.5963513851165771, 1.8111279010772705, -0.4600681662559509, -0.483058899641037, -0.9854729175567627, 1.0364278554916382, 0.4670131802558899, -1.1153995990753174, -0.5636069178581238, -0.8883528113365173, -1.3305563926696777, -1.0627391338348389, -1.1913243532180786, -0.3971749246120453, -0.08661948144435883, -2.290677070617676, 0.48592129349708557, -1.849751591682434, 0.11772649735212326, -1.7091659307479858, 1.8020429611206055, -0.5905373096466064, 0.6578355431556702, 0.9895123839378357, 0.04033232107758522, -0.3781592845916748, -0.1982593536376953, 0.27024465799331665, -1.4178954362869263, -0.9048013091087341, -1.750624656677246, -2.177766799926758, -0.6629294157028198, 0.8174104690551758, 2.6229398250579834, 0.7999602556228638, -1.6758439540863037, -0.038825832307338715], [1.63516104221344, -1.062928318977356, -0.09024132788181305, 0.5967851877212524, 0.18814074993133545, -0.26873961091041565, 0.4315112829208374, 0.13428446650505066, -1.0053672790527344, 0.1856984943151474, 0.283019095659256, 0.45490092039108276, 0.02453591488301754, 0.3462170362472534, -0.29279690980911255, -0.3639431297779083, 0.7971882820129395, 0.8978306651115417, -1.8960938453674316, 0.8320883512496948, -0.6735819578170776, -0.9992551803588867, 1.298500657081604, -0.7814345359802246, 0.19507259130477905, -0.9902603626251221, -0.8299707174301147, 0.617668628692627, -0.12302634865045547, 1.864790916442871, -0.2666895389556885, -1.712693452835083, -0.6621774435043335, -0.10178810358047485, -0.9045552611351013, -1.1803983449935913, 1.3873519897460938, 0.7775408029556274, 1.1572431325912476, 1.4775974750518799, -1.7447116374969482, 1.2253012657165527, -0.6109614372253418, -0.6470556259155273, -0.812417209148407, 0.3697018027305603, -0.07131029665470123, 1.7786240577697754, 0.09650633484125137, -0.23175980150699615, -0.7026238441467285, 0.8801094889640808, -0.779193639755249, 0.8838181495666504, 0.9145346283912659, 0.17680902779102325, -0.7391406297683716, 0.055574048310518265, -0.08285082131624222, 0.07975603640079498, -0.662909746170044, -0.36915311217308044, 2.308065176010132, -1.939345121383667, 0.4999712407588959, 0.03323221206665039, 1.2601488828659058, 2.3646790981292725, 1.204315185546875, 1.3957133293151855, 0.8160150647163391, -0.6817594766616821, 0.12276554852724075, -3.3832318782806396, -0.6719528436660767, 0.0736052617430687, -1.0579707622528076, -0.030088527128100395, -1.2747623920440674, 0.5058785080909729, 1.2296894788742065, 0.6305764317512512, -1.7028497457504272, -0.37312501668930054, -0.10845832526683807, 1.890594482421875, -1.5358989238739014, -0.7703800797462463, -2.2619848251342773, 0.27454349398612976, -0.3560258448123932, 0.28738218545913696, -1.5065816640853882, 0.422448068857193, -0.9837614297866821, 0.51961350440979, 0.2214270383119583, 0.5105563402175903, 1.0224944353103638, -0.8130980134010315, -1.6475859880447388, -0.33016687631607056, 0.3616315722465515, -0.7343505620956421, 0.5336544513702393, -0.6330906748771667, 0.8591064810752869, -0.1918151080608368, 0.8792174458503723, 0.055883534252643585, 0.418077677488327, 0.7648632526397705, 0.09439516067504883, -1.6888127326965332, -1.1711558103561401, -0.5309813618659973, 0.6718255281448364, 1.831393837928772, -1.3940367698669434, -1.0180307626724243, -0.047264743596315384, 0.10807663947343826, -0.07613164931535721, 0.9299651384353638, 2.590118646621704, 0.45323875546455383, -1.2598464488983154, -1.3737645149230957, 0.07740884274244308, 1.168113112449646, -0.17532140016555786, -2.5981602668762207, 0.7071499824523926, -0.4659844636917114, 1.0424057245254517, -1.2860428094863892, 0.04501231014728546, 0.016441166400909424, 0.43745818734169006, 0.763695478439331, -0.5343455076217651, 0.7727146744728088, -0.06378927081823349, 1.4149065017700195, 0.6199079751968384, 0.2515306770801544, 0.11724056303501129, 0.7870694994926453, 1.3394389152526855, 0.915610671043396, -0.6847402453422546, -2.2615885734558105, 1.9785200357437134, -1.9378507137298584, -0.23753778636455536, -1.6817313432693481, -0.20903432369232178, -0.24308140575885773, -0.5664544701576233, 0.5053315162658691, -0.7407503128051758, -0.9043214321136475, -1.414723515510559, -1.5320245027542114, -0.24784111976623535, -0.931900680065155, 0.7567341327667236, 1.7889653444290161, -0.4407195448875427, -1.2334370613098145, -0.16192765533924103, 1.3421826362609863, 0.5072146058082581, 0.8699749708175659, -0.36768966913223267, -0.17389805614948273, 1.6408663988113403, -0.9507943391799927, 0.543143093585968, 0.41081658005714417, 1.0383868217468262, 1.7402721643447876, 0.571148157119751, -1.1145216226577759, -2.6560423374176025, 0.37708908319473267, 1.8827078342437744, 0.11729001253843307, 0.011810495518147945, -1.0153907537460327, -1.6229575872421265, 0.9533803462982178, 1.2098987102508545, 0.22404976189136505, 0.3088095486164093, -0.04651445150375366, -1.8196452856063843, 0.30047401785850525, -0.34175747632980347, -1.0698480606079102, -0.19783298671245575, -0.21243207156658173, 0.4732334613800049, -0.18490536510944366, 0.8644658327102661, -0.5269895195960999, -1.132340431213379, -1.8662283420562744, 0.4769594371318817, -1.403286099433899, -0.03832259029150009, -1.1344414949417114, 0.16461604833602905, -1.8043293952941895, 1.3599661588668823, 0.1713116317987442, -0.607677161693573, 0.22740280628204346, -0.24962440133094788, -0.6856755018234253, 0.5516005158424377, 1.6095257997512817, -0.12093031406402588, 0.6945358514785767, -1.6743277311325073, 1.0685927867889404, -0.8053753972053528, -0.7692922353744507, -1.3761073350906372, -0.3523921072483063, 1.4804420471191406, -0.5319238901138306, -0.8602456450462341, -0.45981863141059875, 0.17467539012432098, -0.3198520839214325, -0.07840336859226227, -2.064277172088623, 1.357021689414978, -0.1407138556241989, 0.3024732172489166, 0.8596329689025879, -0.7528220415115356, 0.6184237003326416, 1.7542662620544434, -1.0598015785217285, 0.7474400401115417, 0.0845310389995575, 0.9892805814743042, -0.3002179265022278, -1.0900391340255737, -0.9874485731124878, -1.8827286958694458, 0.8812974095344543, -0.36017242074012756, 2.233943223953247, 0.9362563490867615, 0.560218095779419, 0.5235381722450256, -0.4907691180706024, 2.3807201385498047, 1.6925592422485352, -0.27896589040756226, 1.2712194919586182, -0.8199463486671448, -0.4491356611251831, -2.3105239868164062, -0.34534773230552673, -2.1352155208587646, -1.047464370727539, 0.016210028901696205, 0.7697694897651672, -0.48691368103027344, 0.8917277455329895, 0.5090699791908264, -0.639142632484436, 1.2217577695846558, 0.8299223184585571, 0.4148527681827545, 1.8605015277862549, 0.3900988698005676, -0.5777931809425354, 0.7556655406951904, 0.7877336740493774, -0.7463282346725464, 2.1269443035125732, -0.8066040277481079, 0.3403538167476654, -0.317850261926651, -1.2919989824295044, 0.10173674672842026, 0.2530941963195801, -0.11095425486564636, -0.6709180474281311, 1.9474107027053833, 1.5698996782302856, 0.06841987371444702, -0.7261505126953125, -2.210491895675659, 0.9106642007827759, -1.8270398378372192, 0.5095696449279785, -0.7369782328605652, 0.6708161234855652, -1.01972496509552, 1.5258827209472656, 0.1265507936477661, -1.0056835412979126, -0.45647743344306946, -1.3260128498077393, 0.37663695216178894, 1.6377363204956055, -0.5265949964523315, 0.29987454414367676, -1.644220232963562, -0.36738917231559753, 0.4078948199748993, 0.3162473142147064, 1.8385387659072876, 0.7945016622543335, 0.9103145599365234, 1.076276421546936, -0.06464484333992004, -1.9367845058441162, 0.768298327922821, 0.002675809198990464, -0.897702693939209, 0.3497961759567261, -1.0904841423034668, -1.3067806959152222, -1.7105878591537476, -1.3534739017486572, 0.35402292013168335, -0.7760154604911804, -0.19711856544017792, 0.6961305737495422, 0.9335796236991882, -0.3487541973590851, 0.3332870602607727, -0.0509214773774147, -0.27440401911735535, -1.4471487998962402, 0.667923629283905, 1.2949398756027222, -0.1010747030377388, -0.9669259786605835, 1.6364878416061401, -0.40735572576522827, -1.9634504318237305, -0.17873714864253998, -1.580283522605896, 0.6893566846847534, 0.47036483883857727, -0.1752568483352661, 1.6991071701049805, -0.12166095525026321, 0.4852195680141449, 0.3095724284648895, -0.06350372731685638, 0.34555214643478394, 0.15758955478668213, -0.5324427485466003, -1.4668937921524048, -0.4647650718688965, -0.6587256789207458, 1.508966088294983, 0.6726986169815063, -2.660332441329956, -1.253429651260376, -0.2732141315937042, -0.1480032056570053, -0.272372305393219, -0.6518713235855103, -2.310685873031616, -2.2002127170562744, 0.2463073879480362, 1.7682428359985352, -0.4790566563606262, -1.196897268295288, 0.6013554334640503, 0.6890779137611389, 0.16279926896095276, -1.067982792854309, -0.6899623274803162, -1.3139539957046509, 0.41918542981147766, 0.32456108927726746, -1.2335951328277588, 1.3772832155227661, 0.43017223477363586, 0.7311908006668091, -0.8953320384025574, -0.4942846894264221, 1.8177120685577393, 0.506737232208252, 0.2905642092227936, -0.48171281814575195, 0.05383908003568649, 1.514426350593567, 1.0610448122024536, -0.49149012565612793, 0.9126218557357788, 0.9663015007972717, -0.997753381729126, 0.739389181137085, 0.7997434139251709, 0.20674391090869904, -0.7740287184715271, -0.18086408078670502, -0.18021322786808014, 1.826062798500061, 2.2039103507995605, 2.1957178115844727, -0.6108483672142029, -0.058222316205501556, -1.193294882774353, -0.341533362865448, -1.173563838005066, 1.9450337886810303, -0.2759767472743988, -0.3837854266166687, -1.6770508289337158, -0.6087325215339661, -0.6430975794792175, -0.5056559443473816, 0.693386971950531, -0.388840913772583, 2.4590861797332764, -0.529415488243103, 0.8901697397232056, 1.1166138648986816, -0.3192363381385803, -0.17774242162704468, 0.5743610858917236, 0.8616805076599121, -0.8104296922683716, -0.38596731424331665, 2.053664445877075, 1.4514567852020264, -0.2239161729812622, -1.6973129510879517, -0.16663117706775665, -1.133384108543396, 0.29164642095565796, 0.6711248159408569, -1.159441590309143, -1.0136631727218628, -1.3005014657974243, -1.2048577070236206, 0.02089138701558113, 0.4565581977367401, 0.5829896330833435, 0.2670813202857971, -0.17176152765750885, 0.4244353175163269, 1.842158317565918, -0.4321116507053375, 1.1565449237823486, 0.679385244846344, 1.0892179012298584, 0.8695884346961975, -1.07106351852417, 1.6196881532669067, -0.7492167949676514, -0.4762583076953888, 0.009899037890136242, -1.5829267501831055, 0.8932034969329834, 0.35811305046081543, 0.12492340058088303, 0.8259890079498291, 2.5248937606811523, 1.1110457181930542, -0.8928351998329163, -0.7929412126541138, 0.5474432110786438, 0.5659192800521851, 0.5992592573165894, -0.1981462985277176, 1.9777436256408691, -1.1553208827972412, 0.3433894217014313, 0.8634358048439026, 0.8861903548240662, -0.04901612922549248, -2.0899174213409424, 0.42165493965148926, 0.13958491384983063, -2.505540370941162, -0.1287965029478073, -1.0286864042282104, -0.3961985409259796, -0.17750532925128937, 0.36068129539489746, 0.21249642968177795, 1.0410778522491455, 0.054548777639865875, -0.16217128932476044, 0.06717041879892349, 1.0938336849212646, 0.16522157192230225, 1.14274263381958, -1.8166025876998901, 1.1170988082885742, 1.248247742652893, -0.26565366983413696, -0.09303966909646988, 0.9296454191207886, 1.9184513092041016, -0.3704979121685028, -0.543972373008728, 0.32352179288864136, 0.3894464075565338, -1.1982192993164062, -0.5800569653511047, -0.21141870319843292, -0.266345739364624, -0.7320893406867981, 0.34855109453201294, -1.052453875541687, -0.2310490757226944, 1.6361314058303833, -0.10527856647968292, 0.32313087582588196, 1.8343294858932495, 1.4735894203186035, 0.09401363134384155, 0.2715652287006378, -1.3032208681106567, -1.1731139421463013, 0.19381199777126312, -0.7450800538063049, 0.7258282899856567, -0.07539928704500198, 1.4247292280197144, 0.5187499523162842, 1.3918728828430176, 0.7582646608352661, -0.47029632329940796, 0.19348889589309692, -0.8776576519012451, 1.437597393989563, -1.3157063722610474, 1.03530752658844, 0.3346530497074127, -0.827041506767273, 0.0238704364746809, 0.4445480704307556, -3.173194169998169, 0.9491854906082153, -2.0604300498962402, -0.11370645463466644, -0.9129674434661865, 0.030990073457360268, -2.299980640411377, 0.24092063307762146, 1.1506305932998657, 0.8366060853004456, -1.126177191734314, 0.2219225913286209, 0.2600789964199066, 0.28633803129196167, 0.4197416603565216, -0.37510794401168823, -0.5000631213188171, 0.5360739827156067, 2.383925437927246, 1.2003726959228516, 1.361474633216858, -0.4252910614013672, -0.6863305568695068, -0.9226370453834534, -1.4711865186691284, -1.086867332458496, -1.064302921295166, -1.0028728246688843, 0.9297180771827698, -1.1071035861968994, -0.8376477956771851, -0.03172418102622032, 1.108794927597046, -1.5876957178115845, 2.375447988510132, -1.5662202835083008, -0.2416279911994934, 0.1494399905204773, -1.3610843420028687, 0.20261207222938538, 1.898802399635315, -0.834298312664032, 0.5606469511985779, -0.5746157765388489, -1.6454333066940308, 1.4582569599151611, -0.7654894590377808, 1.7850985527038574, 0.26958245038986206, 0.9528613686561584, -1.3357064723968506, -1.4283732175827026, -0.9705961346626282, -2.127995252609253, 0.10545776784420013, 0.31637129187583923, -0.1806344836950302, -0.8994067311286926, -0.08401454985141754, -0.4244104027748108, -0.8567818403244019, 0.4189040958881378, 1.6278225183486938, -2.0723721981048584, -1.0526642799377441, 0.23376446962356567, -0.09141965955495834, -0.10973118990659714, -0.873712956905365, -1.4058018922805786, -0.21465390920639038, 0.10558176040649414, 0.5237243175506592, -0.18807198107242584, 0.18409918248653412, -1.7816137075424194, 0.8215476870536804, -0.25421687960624695, -0.6423659920692444, -0.8292452096939087, 0.48223087191581726, 0.23644451797008514, -0.512840986251831, -1.2455053329467773, 0.2294425368309021, -0.47876614332199097, -0.3673662543296814, 1.0373018980026245, -0.8349366784095764, 2.1052560806274414, -1.2303681373596191, -1.4189426898956299, 0.6387160420417786, -1.3228036165237427, 0.17429649829864502, 0.13794374465942383, -0.5987246632575989, -0.3393454849720001, 1.7082833051681519, -1.2467751502990723, 0.9219806790351868, 0.2238617092370987, -0.027611538767814636, 0.409546822309494, -1.9406607151031494, -0.639128565788269, 0.3451652526855469, -0.5020248293876648, -0.3083411157131195, 0.6003344058990479, 1.0609678030014038, -0.5427744388580322, -1.153136134147644, -1.5267138481140137, 0.44266477227211, -0.7360697984695435, 1.583319902420044, 0.8348588943481445, -0.17477338016033173, -2.237689971923828, 0.3355175852775574, -1.4850420951843262, -0.9217070937156677, -1.1270068883895874, 0.11571658402681351, 1.3169026374816895, 0.03530731424689293, 0.558599591255188, -1.2903496026992798, -1.7445077896118164, 0.19468340277671814, -0.9210368990898132, -0.0693565085530281, -0.7121156454086304, -0.9254415035247803, -2.3650734424591064, -0.08831100910902023, -0.1810014694929123, 0.957731306552887, 0.33692899346351624, 1.7950446605682373, 0.17990122735500336, 1.7116388082504272, 0.9296819567680359, -0.679400622844696, 0.7462325692176819, -0.9598780274391174, 0.7996078729629517, -0.8950775265693665, 0.23925745487213135, 0.14966674149036407, 0.8027093410491943, -0.15265782177448273, 0.03389300778508186, 0.03809935599565506, 0.32680055499076843, 1.268311619758606, -1.033079981803894, -0.5191353559494019, -0.09664664417505264, 0.15167132019996643, -0.7497028112411499, -0.7472027540206909, 1.7927757501602173, 0.4348735511302948, -1.2679190635681152, 0.6749023199081421, 0.20953893661499023, 0.1864219605922699, 0.5308064222335815, 1.5347994565963745, -1.2714403867721558, -0.39703917503356934, 0.2862192392349243, 0.8508265614509583, 0.8199726343154907, -0.8144823908805847, 0.9402778148651123, 0.29145312309265137, -0.08258342742919922, 0.2383463829755783, 1.4130550622940063, 1.313118815422058, -1.7548261880874634, -0.8885048627853394, -0.6428354382514954, 0.11932476609945297, -1.1632848978042603, -0.2561146914958954, -0.1518964022397995, 2.1418819427490234, 0.3250317871570587, -2.3998725414276123, -1.055392861366272, 0.4794456362724304, -0.6371777653694153, 0.4333723187446594, -0.877975583076477, -0.41495606303215027, 0.4890604019165039, 0.032760411500930786, 1.2895474433898926, -1.426795482635498, 0.6249583959579468, -0.37833479046821594, -0.5951707363128662, -0.9097071886062622, 1.9225423336029053, 0.5162225961685181, -0.04231182485818863, 0.959611713886261, -0.3847232162952423, -1.7244348526000977, -0.936105489730835, 0.565083384513855, 0.09684969484806061, 0.09677889198064804, -0.06598561257123947, -0.9009019136428833, 1.8305988311767578, 0.8957740664482117, 0.6758394241333008, 1.5995087623596191, -0.5636721253395081, 1.2768409252166748, 1.4870434999465942, -0.41085827350616455, -1.127774953842163, 1.9696284532546997, 0.55116206407547, 1.7438780069351196, 0.974860668182373, -0.3781283497810364, 0.4711650609970093, 0.8674494028091431, -0.1276838332414627, 0.19955934584140778, -0.303447961807251, 1.4608488082885742, 0.7379563450813293, 0.5786562561988831, -0.6274888515472412, -1.6286883354187012, -0.4739972949028015, 0.36007800698280334, 0.15602611005306244, -0.8478240370750427, -0.4225895404815674, -0.05363614857196808, -0.30244436860084534, 0.18816231191158295, -0.13536782562732697, -0.8908035755157471, 2.597538948059082, 0.9662179350852966, -0.6387940049171448, -1.4868226051330566, 1.1989688873291016, 1.6246129274368286, -0.5277462601661682, 0.7286111116409302, 0.311921626329422, 0.26173529028892517, -1.5604286193847656, -0.3069080710411072, 0.03296492248773575, -0.24849197268486023, -1.6208503246307373, 0.616264283657074, 0.2730458080768585, -0.024888528510928154, 1.4191957712173462, 0.35026872158050537, -0.11402669548988342, -1.9188055992126465, 0.46738770604133606, 0.05515838414430618, 1.5915197134017944, 0.07898551225662231, -0.20843105018138885, 0.2990245223045349, 1.475581169128418, -1.0383061170578003, 1.2673646211624146, 0.6268737316131592, 0.5680072903633118, -1.5867496728897095, -1.4829237461090088, 0.11704146862030029, 0.9250150918960571, -0.3248600959777832, -0.644159734249115, 0.7178037762641907, 0.06005840003490448, -1.1183327436447144, -0.2992103695869446, 1.0797516107559204, 0.7628931403160095, 0.9395338892936707, 2.279053211212158, 1.004526138305664, -0.0903155505657196, 0.9843860864639282, -0.2573815882205963, -0.6591570973396301, 0.8591908812522888, 0.23182784020900726, -0.4614202678203583, -0.4185308814048767, 1.0239015817642212, -0.07056137919425964, -0.6906823515892029, -0.9837870597839355, -0.46691596508026123, 0.31282320618629456, 0.5084794163703918, 2.1841135025024414, 0.5770658254623413, -0.4785211980342865, -1.5904561281204224, 0.4104937016963959, 0.3311619460582733, -0.309283584356308, -0.8631376624107361, 0.493680864572525, 0.650242030620575, 0.6174650192260742, 0.39441561698913574, -0.6412498950958252, -1.0192519426345825, 0.221533864736557, 1.0113261938095093, 0.6036151647567749, 1.9502981901168823, -0.25433313846588135, -0.5197508931159973, 0.7849926352500916, 1.345671534538269, 0.06264062225818634, 0.13493217527866364, 0.15813465416431427, -1.0455769300460815, 0.9540975689888, -1.1616947650909424, 1.0369722843170166, 0.4934943914413452, -0.6464157104492188, -1.2444736957550049, 0.4468490183353424, 0.44517776370048523, -0.10669773817062378, -0.7337853312492371, -0.56255704164505, -0.536194920539856, -1.9680744409561157, -0.0213283970952034, -0.4181082248687744, 0.8175791501998901, -0.13915666937828064, -1.7171868085861206, 0.8624054789543152, 0.570698618888855, -0.25887638330459595, -1.496488094329834, 0.4756123721599579, -0.35141444206237793, 1.0867578983306885, -1.131451964378357, 0.6457703709602356, -0.12309694290161133, -0.4331653118133545, -0.6934540867805481, 0.50943922996521, 1.1670050621032715, 1.033988118171692, 1.2432540655136108, 2.3122804164886475, 0.0606190450489521, -1.9876303672790527, 0.029998820275068283, -0.5102618336677551, -0.9512439370155334, -0.47066760063171387, -0.35917967557907104, 0.06566111743450165, 0.3138658404350281, -0.14758186042308807, -0.5563270449638367, -0.9425455927848816, 0.23278264701366425, -0.027347799390554428, 1.6428005695343018, -1.0777801275253296, -0.8776881694793701, -2.035299301147461, -0.32819727063179016, -1.7847949266433716, 1.2387794256210327, 0.12294498085975647, -0.2518368661403656, 0.043861400336027145, -1.7647724151611328, -0.39818981289863586, -1.6663075685501099, -0.5398258566856384, -0.3310508728027344, -0.9431378841400146, -0.9546602964401245, -0.9742696285247803, -0.4409042000770569, 0.1770995706319809, -1.8828637599945068, -0.629611611366272, 1.408396601676941, -0.8520421981811523, -1.1283130645751953, 0.0742487832903862, 0.5871706008911133, -1.131485939025879, -0.3825904130935669, -0.19867175817489624, -1.0362249612808228, -0.27601829171180725, 0.537140429019928, -0.17776770889759064, -1.8135439157485962, -1.6006733179092407, 0.023576876148581505, 0.04577132686972618, -1.6248530149459839, 1.1636700630187988, 0.3051545023918152, -0.5110476016998291, 1.1090846061706543, 1.366884708404541, 1.7291548252105713, -0.20551089942455292, 0.5424609780311584, -1.308391809463501, -1.2193725109100342, -0.9880768656730652, 0.8465918898582458, -0.35242918133735657, -1.1811739206314087, -0.7991127371788025, -1.4417738914489746, -1.7632863521575928, -1.0021852254867554, -1.005734920501709, -0.47443196177482605, 0.8204631805419922, -0.4674367606639862, 0.31407803297042847, 0.039361998438835144, -0.4303598701953888, -0.47886449098587036, -0.06280028074979782, 1.4782336950302124, 1.914363980293274, 0.07169494032859802, -1.863196611404419, 0.42414477467536926, 1.5857778787612915, -0.26636239886283875, -0.4164438843727112, 0.13590310513973236, -1.2865220308303833, 2.2180609703063965, 1.3154966831207275, 0.023590534925460815, 0.6256800293922424, 1.410134196281433, -1.4647104740142822, -1.7042042016983032, -1.340367317199707, -0.7413846850395203], [4.4231109619140625, -0.5178433060646057, 2.299089193344116, -1.0892090797424316, 0.6794121861457825, 0.603209912776947, 0.00661513302475214, -0.48493343591690063, 0.7654873728752136, 0.5677153468132019, -2.289085626602173, 0.4746568202972412, 0.5768674612045288, -0.7255907654762268, 0.1553090363740921, -0.4097093641757965, -0.27248942852020264, -0.5392747521400452, -1.0680772066116333, -1.7556339502334595, 0.16086705029010773, -0.20920337736606598, 0.5939412117004395, -1.42795729637146, 0.6875045299530029, -0.5839714407920837, 0.3829355835914612, 2.0003671646118164, 2.531841278076172, 0.3029729127883911, 0.7153856158256531, 0.4632553458213806, -1.5309011936187744, -0.2812001705169678, 0.8941605687141418, 0.3501615524291992, 2.3910059928894043, 0.484768807888031, -0.7091963291168213, -1.5284734964370728, 0.7914815545082092, 0.7747786641120911, 0.3301781415939331, 0.1936853975057602, 0.13580353558063507, -0.7427650690078735, -0.3647348880767822, -1.7844442129135132, 0.7944716811180115, -0.5097729563713074, -1.1493138074874878, -0.5884068608283997, -0.2346504181623459, -0.14317528903484344, 0.3500536382198334, 0.6433655619621277, 0.5253402590751648, -0.8767347931861877, 0.44036728143692017, 1.234488844871521, -0.41276976466178894, -0.7219658493995667, -0.26601341366767883, 2.361327886581421, 1.8086936473846436, 1.8220001459121704, 1.6221791505813599, 0.5493035316467285, -0.02543978951871395, -1.8512651920318604, 0.13347956538200378, -1.574662208557129, 0.3021065592765808, -0.5533688068389893, -0.48117873072624207, -0.7069281935691833, 0.9098671078681946, -1.4445836544036865, -1.46250581741333, -0.2267610728740692, 1.236586332321167, -1.6451600790023804, 0.4894666075706482, 0.9780261516571045, -0.5957342982292175, 0.9519913792610168, 0.9347624182701111, 1.0002225637435913, 1.4082262516021729, -0.5152770280838013, 0.8204526901245117, -0.9589369893074036, 0.05417855829000473, 1.2901320457458496, -1.3310577869415283, 0.46509796380996704, 0.1542290449142456, -0.566600501537323, 0.845054030418396, -1.326896071434021, 1.3373337984085083, -1.1895360946655273, 0.6042876243591309, 0.8140232563018799, 2.467587947845459, 0.49648353457450867, -1.5794219970703125, 0.513846218585968, -1.080252766609192, -0.3152695298194885, -0.41445356607437134, -0.015289213508367538, -0.7502697706222534, 0.8601923584938049, -0.6354876756668091, -0.2251550704240799, 0.9102655053138733, -2.150700807571411, 1.829319953918457, 0.26136070489883423, 0.5058090090751648, 0.7453904151916504, -0.4908076524734497, 0.27251631021499634, -0.88580721616745, 0.6721230745315552, -0.800007164478302, 0.3329260051250458, -2.0493366718292236, 1.3300585746765137, -1.509257435798645, 1.1122475862503052, 0.8059971332550049, -0.6192651987075806, 0.2581052780151367, 1.1579489707946777, 0.6593021154403687, 0.8875946402549744, -1.504636287689209, -1.423069953918457, -1.046081781387329, 0.6321589350700378, 1.515851616859436, 0.48760488629341125, 1.3343294858932495, 1.1156718730926514, -1.110486626625061, -0.7416889667510986, -0.6427876949310303, -1.4259506464004517, -0.7580808401107788, 0.2897491157054901, -0.31842461228370667, -0.24501414597034454, -0.3267706334590912, 0.6517280340194702, 0.3817501366138458, -0.5304393172264099, 0.6066609621047974, 0.6651314496994019, -1.376335859298706, 1.3539338111877441, 0.6058204174041748, 2.7168798446655273, -0.3309716284275055, 1.008920669555664, 1.0094224214553833, -1.3934924602508545, 1.2466063499450684, -1.3218525648117065, 0.19465233385562897, 0.8395525217056274, -0.9203265905380249, 0.11039102077484131, -0.31228354573249817, 0.6281811594963074, 2.0480668544769287, -0.1312905251979828, -0.6015328168869019, -1.582214593887329, 0.00593061838299036, -0.6595675349235535, 0.04022728651762009, -1.8563824892044067, -0.4014017581939697, 0.11992134898900986, -0.6179410219192505, 0.6544637084007263, 0.36545395851135254, -0.24329449236392975, 0.06780413538217545, -1.8574373722076416, -0.23147641122341156, 0.36216315627098083, -0.9061932563781738, 1.2721309661865234, 1.9634751081466675, -0.2524271309375763, 0.44381245970726013, 0.9641779661178589, 0.4054664969444275, -0.3220366835594177, -0.3504481017589569, 0.6154371500015259, -0.46584755182266235, 0.7776981592178345, -0.4141872525215149, 0.26708516478538513, -1.4495658874511719, 0.8981854319572449, 0.4312763810157776, -0.6471213102340698, 0.37383052706718445, 0.5465220808982849, 0.08393402397632599, 1.1839145421981812, 1.7211910486221313, 0.4519520699977875, 1.9631518125534058, 1.175638198852539, -0.2078941911458969, 0.401071697473526, 1.6348023414611816, 1.5461105108261108, 0.9057531356811523, 0.10620957612991333, -0.6160875558853149, -0.10091765969991684, -0.7813218235969543, 0.44818025827407837, 0.5774540901184082, 0.052128687500953674, -0.1450483351945877, -0.7085416316986084, 0.7899145483970642, -1.0661760568618774, -0.7014856934547424, 0.714217483997345, 0.18144333362579346, 0.4272705912590027, -0.3540365695953369, 0.046715397387742996, 1.925991415977478, -0.7799197435379028, 0.555385172367096, -0.7782822251319885, 0.8209399580955505, 0.7579246163368225, 0.032679568976163864, 0.3803504705429077, 1.5336767435073853, -0.7610859274864197, 2.0094778537750244, 0.8967348337173462, 1.6273510456085205, 2.3541224002838135, -2.9505977630615234, 1.578187108039856, 0.19689203798770905, 0.24922750890254974, -0.9636337161064148, -1.9680159091949463, -0.8900101184844971, -1.3442590236663818, -2.01694393157959, -0.7201445698738098, 0.21598638594150543, -0.21596477925777435, 0.26874905824661255, 0.9879611134529114, -0.15944641828536987, -0.8835312724113464, -0.05954768881201744, 6.722936814185232e-05, -0.41392987966537476, 1.1226054430007935, -0.014305319637060165, -0.2913959324359894, -1.5265637636184692, 0.4219483435153961, 0.17176760733127594, 0.4793071746826172, 1.268694281578064, -1.5798399448394775, 2.240591049194336, -0.04845826327800751, -0.3251824378967285, 0.3648732006549835, -1.315398931503296, -1.3038145303726196, -2.0650851726531982, 1.4005340337753296, -0.24278707802295685, -0.20387263596057892, -0.8342706561088562, -0.7812351584434509, -1.75919508934021, -0.6128089427947998, -0.16006708145141602, 0.5680907964706421, -1.2843126058578491, 0.5326343178749084, 0.43791815638542175, 0.5428038239479065, -0.14133115112781525, -0.06331200897693634, -0.7913188934326172, 1.269364356994629, 1.4991124868392944, -0.5165923833847046, -0.3888552486896515, -0.390525758266449, -0.3599553406238556, -0.09326373040676117, -0.2916369140148163, 0.9545764923095703, 1.670472264289856, -0.8960682153701782, -0.12839137017726898, -0.18735024333000183, 1.1595090627670288, 0.3971983790397644, -0.8621575236320496, -0.44690102338790894, -0.383539617061615, -0.5559662580490112, 1.876317024230957, 1.1606082916259766, 0.5463202595710754, 0.709571897983551, 1.3220139741897583, 0.6905602812767029, -1.4890846014022827, 0.3496493995189667, -0.1060672253370285, -1.3659816980361938, 0.2408703863620758, -0.8420798778533936, -0.14391010999679565, -0.568367600440979, -0.8085695505142212, -0.14757291972637177, 0.43450289964675903, 1.1441116333007812, -0.671240508556366, -0.48277148604393005, 1.0599141120910645, 0.8705675005912781, 0.7005479335784912, -0.1946350634098053, 0.6018142700195312, -0.47017902135849, -1.5114386081695557, 0.4702672064304352, -0.27957433462142944, -0.5377570986747742, -0.13795457780361176, 2.0712084770202637, -1.2264002561569214, 0.14501294493675232, 3.0188450813293457, 0.6122123599052429, -1.9815815687179565, -0.1957932561635971, -0.5158621668815613, -0.4870533049106598, -0.4265781342983246, 0.552653968334198, 1.9182546138763428, 0.6203486323356628, 1.4801090955734253, -0.03044271469116211, -0.10858070850372314, 0.5302966237068176, -0.4515742063522339, 0.7147650718688965, -0.9251080751419067, 0.5843003392219543, 1.1015090942382812, 0.02007516846060753, -0.13923318684101105, -1.5801775455474854, -0.8499792218208313, -1.9167248010635376, 1.5956308841705322, -0.26564595103263855, -0.6549460887908936, 0.3978651165962219, 1.4157778024673462, -0.01638271100819111, -0.3274284601211548, 0.9867894053459167, -0.139229878783226, -1.7375057935714722, 0.25647419691085815, -0.45695799589157104, 0.945434033870697, -0.37129855155944824, -0.07339128851890564, 1.590461254119873, 0.9247862696647644, 0.8628838658332825, 0.5255342721939087, -0.1951063573360443, 0.36588412523269653, 0.27805420756340027, 0.9749573469161987, -2.2251570224761963, -0.9321693181991577, -0.7432385683059692, -1.769260048866272, 0.7284230589866638, -0.041213508695364, 1.1810959577560425, 0.7127918004989624, 0.7823656797409058, -1.1334030628204346, -1.3081457614898682, 0.7485154867172241, 0.5183860659599304, 0.3759239614009857, -0.9862126708030701, 1.217612385749817, 0.12835028767585754, 0.0754842534661293, 0.29367268085479736, -0.20396015048027039, -1.9645323753356934, -1.6520392894744873, 0.07711969316005707, 0.47908568382263184, -0.6521916389465332, -1.2327017784118652, 0.5005113482475281, -0.01654195971786976, 0.15188980102539062, -0.434068888425827, 2.8530709743499756, 0.29647010564804077, -0.7024083733558655, -0.06522880494594574, -0.43697547912597656, -1.0214250087738037, -0.3588152229785919, -0.03354157507419586, -1.2699424028396606, -0.6264825463294983, -1.1941516399383545, -0.3657718300819397, -0.5681560039520264, -0.07737525552511215, -1.060065507888794, 0.0291737150400877, 0.4129229784011841, -0.3492528796195984, 1.5234973430633545, 0.498474657535553, 0.3746035397052765, -0.07294201850891113, 0.47266721725463867, -0.731309711933136, -0.8482171297073364, -1.2919119596481323, 0.1917901486158371, -0.4119836390018463, -0.8020722270011902, 0.8050664663314819, -0.898066520690918, 1.501763105392456, 0.2539866268634796, -0.5995002388954163, 1.301813006401062, -0.5353800058364868, 0.6416414976119995, 1.2942686080932617, -1.2343579530715942, 0.010490724816918373, -1.7355830669403076, -0.5645503401756287, 0.18219208717346191, 0.30474647879600525, 1.1416735649108887, 0.43851804733276367, 2.827322244644165, -0.1112995520234108, -0.46710124611854553, -1.4039289951324463, 0.47527050971984863, -0.11162320524454117, -1.2135069370269775, -0.5410311818122864, -0.2648250162601471, -0.5265054702758789, 1.0239697694778442, -1.314476490020752, -1.2905136346817017, 0.2605200707912445, 1.3891323804855347, -0.31749746203422546, -0.7373231649398804, -0.7674366235733032, 0.49041587114334106, -0.05560661479830742, -0.7305504679679871, -2.303194761276245, -0.512040376663208, -0.1294550895690918, -1.32383131980896, 0.4015581011772156, 0.24942399561405182, -2.174908399581909, -0.44249579310417175, -2.4013967514038086, 0.5028425455093384, -1.2614156007766724, 0.07446768134832382, -0.8375513553619385, 0.8937220573425293, -0.625970184803009, -0.8582388758659363, -0.2661948800086975, -1.243566632270813, 1.3365061283111572, 0.5060144066810608, 0.7170657515525818, 0.3838624060153961, -0.13827966153621674, 1.2934064865112305, 0.21848341822624207, 0.20129717886447906, 0.25328269600868225, -0.5345940589904785, -0.9717208743095398, -0.22902193665504456, -0.01656273566186428, -1.1818907260894775, -0.8039107918739319, 0.04603119567036629, 1.1044119596481323, -1.0107680559158325, 0.1724403351545334, -0.49381035566329956, -0.5848566293716431, -1.4379338026046753, 0.7110345959663391, 0.16482123732566833, -1.6964819431304932, 0.20354795455932617, -1.2533971071243286, -1.504679799079895, -0.7039956450462341, -0.16907355189323425, 0.7281882166862488, 0.3856697976589203, 1.3906347751617432, 1.1695561408996582, -0.4891756772994995, 1.1574467420578003, 0.049032557755708694, 1.398912787437439, -1.5810047388076782, 0.39096590876579285, -0.5837520360946655, 0.6289831399917603, 0.09221388399600983, 0.12043126672506332, 0.21866725385189056, -0.7943025231361389, -0.9586719870567322, 0.5317122936248779, 1.4939930438995361, -0.5902510285377502, -1.1782925128936768, 0.5526095032691956, 2.5996885299682617, 0.3612491190433502, -0.10561791062355042, 0.219532310962677, 0.19675271213054657, -0.9227982759475708, 0.005575122311711311, 0.8271436095237732, -0.6140280961990356, -0.514283299446106, 0.4778369963169098, -0.07788299024105072, -0.36190566420555115, 0.4832313060760498, -0.7492280602455139, 0.07882143557071686, 0.8511790037155151, -0.18155379593372345, 1.6055994033813477, 0.6272566914558411, -0.0779222622513771, 1.6122550964355469, -1.023746371269226, 0.4820083975791931, 0.9973174333572388, 0.5801590085029602, -1.2603273391723633, 1.1614596843719482, 0.553281843662262, 0.9655284285545349, -1.232484221458435, -1.0211541652679443, 0.43991464376449585, 0.5773268938064575, 0.544581949710846, 0.5175336003303528, -0.28026893734931946, -0.06120283901691437, -0.9983661770820618, 1.571743369102478, 0.689972996711731, 1.8655974864959717, 0.30835169553756714, -0.867524266242981, -0.1051085889339447, 0.33918237686157227, 1.3963466882705688, 0.675882875919342, -0.015360617078840733, 0.16560786962509155, -1.0467958450317383, -0.21969875693321228, -0.1005093902349472, -1.1521127223968506, 0.2940673530101776, -1.411857008934021, 0.0057215564884245396, 0.542658269405365, 0.8862236142158508, -0.7532314658164978, 1.1517797708511353, -1.2875877618789673, -1.2019871473312378, 0.9579436182975769, -0.9827858805656433, 0.12585197389125824, -1.0159249305725098, -0.7021318078041077, -0.7879464030265808, 0.1687999814748764, 1.7987271547317505, -0.7812700867652893, -2.1309216022491455, 1.097874641418457, -1.499708652496338, -0.6247145533561707, -0.34892067313194275, -2.44267201423645, -0.3460269570350647, 0.2901655435562134, 1.6699261665344238, 0.7601131796836853, -0.1119767427444458, -1.2523142099380493, -0.44978174567222595, 0.7823866009712219, -0.07150276750326157, 0.40100282430648804, -0.8069921731948853, 0.03057733178138733, -0.6420602798461914, 2.596423625946045, 1.4865005016326904, 1.2812416553497314, 0.6189849972724915, 0.705178439617157, -0.6464709043502808, -1.168250322341919, 0.7633376121520996, 1.9479711055755615, 0.1784975379705429, 1.6703048944473267, -0.6280792951583862, -0.27284517884254456, 0.09632854163646698, 0.03080483339726925, -0.3349955379962921, -1.035423755645752, 0.8527305126190186, -0.7623856067657471, 0.6630542278289795, -1.2862364053726196, 0.424878865480423, -0.42042821645736694, 0.07136552035808563, -0.884288489818573, -0.8107001185417175, 0.07825671136379242, -1.6378570795059204, 0.12564319372177124, 2.0269699096679688, -0.521837055683136, 2.2541816234588623, 0.15772627294063568, 0.026980966329574585, 0.12811598181724548, -0.22689367830753326, 0.6442236304283142, -0.06273829936981201, -0.543077290058136, -0.1390223503112793, 0.27828365564346313, 2.221837043762207, 1.610676884651184, 0.8562520742416382, 2.301711320877075, -0.9884324669837952, 0.7782816290855408, -0.35050344467163086, -0.7342531085014343, -1.4673856496810913, 0.059341393411159515, -1.7539072036743164, -0.1392277330160141, -0.9698723554611206, -0.9555953741073608, 0.11417979747056961, 0.8905564546585083, 2.262866258621216, -0.5029529929161072, -1.0082553625106812, -0.23395641148090363, 1.0246119499206543, 1.745255947113037, 0.5967850089073181, 0.8094285130500793, 0.021894138306379318, 0.8960504531860352, 0.2717953324317932, -1.518141508102417, 0.6657794117927551, 0.6703141331672668, 1.426464319229126, 0.34656277298927307, -0.8275788426399231, 1.2158259153366089, 0.7003040313720703, 1.5539815425872803, -0.942138135433197, 0.32876092195510864, -0.20360210537910461, -1.8105345964431763, 0.1283215582370758, -0.31150078773498535, -0.9304280281066895, 2.189898729324341, 0.6132797598838806, -0.1838144212961197, -1.3004566431045532, -0.7535797357559204, -0.6241761445999146, -1.009595513343811, 0.309406578540802, -0.7127463817596436, 0.775829553604126, -1.2271409034729004, 2.2166242599487305, -0.25433674454689026, 0.3622201979160309, 0.6892847418785095, -1.1371920108795166, 0.40299689769744873, -0.000798362132627517, 0.192546084523201, -0.07306770235300064, 0.9006192088127136, 0.6830547451972961, 1.3471425771713257, -0.8844457268714905, -0.5198615789413452, 0.7827485203742981, -0.34692835807800293, 0.3552539646625519, 1.3771146535873413, 1.2036734819412231, -0.7625563740730286, 0.36082664132118225, -2.1507937908172607, 1.6166805028915405, -1.603419303894043, -1.1097033023834229, -1.4377738237380981, -1.8048433065414429, -2.1524102687835693, 1.6343598365783691, -1.7376196384429932, -0.6137605905532837, 1.772149682044983, 0.20914717018604279, 0.22769618034362793, 1.3819748163223267, 0.22085699439048767, -0.30300581455230713, -0.13259302079677582, -1.0052576065063477, -0.8246088624000549, 0.7845834493637085, -0.5042603611946106, -1.0038331747055054, -0.33991602063179016, 0.20956023037433624, 0.7880528569221497, -0.05374080315232277, -0.9098098278045654, 1.2079437971115112, 0.9605584144592285, -0.1374318152666092, 0.7521248459815979, 0.9938015937805176, 0.11396736651659012, -1.3916409015655518, -0.8415897488594055, 1.795529842376709, 1.7739931344985962, 1.7270859479904175, 1.1323992013931274, 0.15659525990486145, 0.57001131772995, 0.14972715079784393, -0.19581450521945953, -1.3196730613708496, 1.0877972841262817, -0.7962886095046997, -0.662237286567688, -0.14043058454990387, 0.6763211488723755, -0.11239604651927948, -1.5214346647262573, -0.06760310381650925, -0.2789890170097351, 0.44093286991119385, -0.4869271516799927, -0.41416260600090027, -0.799377977848053, 1.2211376428604126, -0.9974120855331421, 0.5994935631752014, 0.07552015781402588, 0.3541046380996704, -0.46356844902038574, 0.5174576640129089, -0.03342890739440918, 0.19096232950687408, 1.6400527954101562, -1.5880529880523682, -1.1502735614776611, 0.670442521572113, -0.5830431580543518, 0.584132730960846, -1.1237674951553345, 0.9119597673416138, -0.2347865253686905, 1.138862133026123, 0.8952269554138184, -2.861616611480713, 0.5674802660942078, 0.3082311451435089, -0.13648925721645355, -1.5963709354400635, 0.4730079174041748, 0.48900485038757324, -1.03685462474823, -0.25133442878723145, -0.22548243403434753, -1.4518988132476807, -1.2048290967941284, -0.4272930920124054, 1.756656289100647, -0.8503780961036682, 1.7960485219955444, -0.0009929131483659148, -0.17119120061397552, 0.24433402717113495, 0.655476987361908, 1.2920852899551392, -1.0285316705703735, -0.4560689926147461, 1.4478110074996948, 0.8799059987068176, -1.6525577306747437, 0.4772655665874481, 1.372463583946228, -0.29500991106033325, 0.4940387010574341, 0.2880943715572357, -0.6893341541290283, 1.0003840923309326, 0.38513872027397156, -0.3492138683795929, -1.1460542678833008, 0.9913545846939087, 0.8831957578659058, -0.581831157207489, -1.7946701049804688, 0.8903836607933044, -2.4188575744628906, 1.9134396314620972, -0.6488951444625854, -0.42220768332481384, -1.4700467586517334, 0.23180106282234192, 0.27335381507873535, -0.2778378129005432, 0.9924824237823486, 0.3346399664878845, 0.6259725689888, 0.954599916934967, 0.268669992685318, -0.7174544930458069, 0.7282633185386658, 1.0213055610656738, -0.8976325392723083, 0.7800801992416382, -1.0349373817443848, 0.8656917214393616, 0.4958963394165039, -2.2649264335632324, -0.4041752219200134, 1.0078383684158325, 1.703243613243103, 0.8496593832969666, -2.7186877727508545, -2.027146816253662, 0.1104554682970047, 0.21065516769886017, -0.5727605819702148, 0.13359501957893372, 1.8785606622695923, 0.12080423533916473, 1.2004952430725098, -0.263163685798645, 1.7246813774108887, -0.8553636074066162, -0.5544691681861877, 1.2924973964691162, -1.2201619148254395, -1.8470134735107422, 2.591867208480835, -0.9781574606895447, 0.3487683832645416, -0.031089002266526222, 0.09275326132774353, -0.8131809234619141, 1.0760976076126099, 2.0396621227264404, -0.8679360747337341, -1.6938430070877075, 1.193353295326233, 0.7028164863586426, 1.0763955116271973, -1.0499335527420044, -2.4069113731384277, -1.1342411041259766, 0.4118049442768097, 0.30076077580451965, -0.9587539434432983, -0.4721538722515106, -0.13125841319561005, -1.8964592218399048, 0.5809665322303772, 0.34263962507247925, 1.16877281665802, 1.4670027494430542, -1.0784993171691895, 0.704716682434082, -1.2674468755722046, -0.030706753954291344, 0.6302423477172852, -1.5083590745925903, 0.19571296870708466, -2.712399959564209, 0.26111575961112976, 1.3937557935714722, 0.35138389468193054, 0.5383460521697998, 0.2312282770872116, -0.4982489049434662, -1.453467607498169, -0.9179434180259705, 1.3249049186706543, 0.2924911081790924, 0.7365202903747559, 1.4843602180480957, -1.0939960479736328, 0.4368457496166229, -1.671669363975525, 0.1128532886505127, 0.17280952632427216, -0.2347254902124405, 0.4058971703052521, -1.4385017156600952, 0.6370560526847839, -1.6846239566802979, 1.075510025024414, -0.24242781102657318, -0.15533488988876343, -0.6055788397789001, -0.4070374071598053, 0.10150028020143509, -0.1541968584060669, -1.8509186506271362, -1.1909939050674438, -0.6486703753471375, 1.13493812084198, -0.7895744442939758, 0.5719938278198242, -0.22863797843456268, 0.596623957157135, 0.3263489902019501, 0.6809293031692505, 0.6964518427848816, -0.6992138028144836, 0.8513292074203491, -0.1861400306224823, -0.08683915436267853, -0.48676493763923645, 2.574639320373535, 0.06255514919757843, 1.213218092918396, 0.9228339195251465], [-1.108926773071289, 1.3698647022247314, -0.09102100133895874, -0.9711279273033142, 0.07817908376455307, 1.0413261651992798, 0.36844709515571594, 0.30432361364364624, -0.8183264136314392, 0.4812270402908325, 0.37718379497528076, 0.9003857374191284, 1.0163861513137817, 0.9538750648498535, 1.2693454027175903, 0.6513690948486328, -0.1977214217185974, 0.3750620186328888, 1.1837797164916992, 1.619064450263977, 1.8669061660766602, 0.20831426978111267, 1.285662293434143, 1.831903338432312, 1.995632529258728, 1.4266561269760132, -0.8376309275627136, 0.2889671325683594, 0.9747680425643921, 0.02115374244749546, -0.7439475059509277, -0.692399263381958, 0.004821165930479765, -0.8389469385147095, 0.706978976726532, 0.22428561747074127, 0.7962626814842224, 0.6433568596839905, -0.05452475696802139, 0.31350141763687134, -0.15250103175640106, -0.4343074858188629, -1.4865041971206665, 1.7120845317840576, -0.04199272394180298, 0.5625978112220764, 1.1733510494232178, 0.4461135268211365, 1.5876108407974243, -1.9361237287521362, 0.3676188588142395, 0.7055287957191467, 0.47544968128204346, -0.47506260871887207, 0.9097821712493896, 0.008493931032717228, 1.5732266902923584, 0.3216390609741211, -1.9068922996520996, 0.9170525074005127, 0.08306803554296494, 0.0012214406160637736, 0.7927954196929932, -0.22368638217449188, -0.13456971943378448, -2.2666635513305664, 0.942743718624115, -0.8944477438926697, -0.6703195571899414, 0.8174731135368347, 0.2337980568408966, -0.3373091220855713, -0.13153646886348724, 0.07708926498889923, -0.6354401111602783, 0.46539372205734253, -1.6755259037017822, -0.2321445643901825, 1.4083046913146973, -1.0483307838439941, -1.0316922664642334, -0.16565148532390594, -1.1690807342529297, 0.17314288020133972, -0.4596588909626007, 0.9545047283172607, 1.0467088222503662, -0.6455221176147461, 0.7354055047035217, 1.1381465196609497, -1.0193456411361694, -0.4201153814792633, 0.25734665989875793, 0.7438554167747498, 2.2407867908477783, -0.5741885304450989, 0.38916927576065063, -1.2729634046554565, -0.4900244176387787, 0.8279284834861755, -0.22163136303424835, -0.5561825037002563, 0.8599967956542969, 0.6286167502403259, -0.8470606207847595, -1.1412699222564697, -0.4614031910896301, 0.4368531405925751, -0.360440731048584, 0.6987865567207336, -0.5624702572822571, 1.0973261594772339, -1.4574978351593018, 0.5671566724777222, -0.3097170889377594, 1.1702098846435547, 1.0238544940948486, 0.8376393914222717, -2.0544636249542236, 0.9688544273376465, -0.15278871357440948, 0.13393737375736237, 0.7752217054367065, -0.12689319252967834, -0.4178139865398407, -0.5015538930892944, -0.034688569605350494, 1.2046244144439697, -0.4496394395828247, -0.8559384942054749, 1.4943406581878662, 0.7232118844985962, -0.03723742812871933, -0.2299962341785431, -0.7214763164520264, 1.243768572807312, -2.5970492362976074, -0.33152449131011963, -0.13822293281555176, -0.9039260149002075, 0.4329419434070587, 0.9234885573387146, 1.3900551795959473, -0.42982879281044006, 0.7756242752075195, -0.44085830450057983, -1.3435001373291016, 1.0106946229934692, 0.6579200625419617, -2.6584112644195557, -0.9009872078895569, 0.6862663626670837, -1.7306921482086182, 1.6553648710250854, -0.7681783437728882, 1.0638071298599243, 0.13725680112838745, 0.6840466260910034, -1.1016770601272583, 0.29990318417549133, -0.2659369111061096, 1.0481157302856445, -0.17195963859558105, 0.9780386090278625, -0.2667802572250366, -0.19338271021842957, -3.7350072860717773, -0.4692404866218567, 0.24667224287986755, 0.7632719874382019, -0.5275624394416809, -0.88254314661026, 0.6924393773078918, 0.0419062115252018, -0.39318934082984924, 1.8991374969482422, 0.4608634114265442, -0.6654102206230164, 1.3585894107818604, -1.0327471494674683, -0.6109833121299744, 1.441818118095398, 0.2941737174987793, -1.1150403022766113, -0.07840859144926071, -0.0919555276632309, -1.3111541271209717, 0.035126298666000366, -0.37749165296554565, 0.28101691603660583, 0.6925471425056458, -0.5394285321235657, 0.5710610151290894, -1.4226425886154175, 0.0016792789101600647, -0.4792252779006958, 1.7701499462127686, 0.5882031917572021, 0.9702029824256897, 1.2198939323425293, 0.3889472782611847, 0.506270706653595, 0.5409247875213623, 0.3942404091358185, 1.1218738555908203, 0.9457046389579773, 0.11588403582572937, 0.024491582065820694, 1.6206434965133667, 1.9024405479431152, 0.6963984966278076, -1.8080435991287231, 0.7588960528373718, -0.8218774795532227, -0.7095080614089966, 1.4770450592041016, -0.2703426480293274, 0.2011590600013733, 0.44678619503974915, 0.6493006348609924, -0.11999040096998215, -0.7227627635002136, -0.8558100461959839, 1.8689745664596558, -0.09937596321105957, 0.04553893581032753, 2.0675532817840576, 0.041352372616529465, -0.0594172403216362, 0.577399492263794, -0.29009994864463806, -2.38608980178833, -1.669079065322876, -1.6436874866485596, -2.5048086643218994, -0.17670926451683044, 0.7208064794540405, -0.5821633338928223, -0.8790841102600098, -1.500848650932312, -0.3610762357711792, 0.12260220944881439, 1.0552443265914917, 0.21725817024707794, 0.44632062315940857, 0.38846898078918457, 0.05193261429667473, -0.08100255578756332, 0.6386085748672485, -1.8567179441452026, -1.5096293687820435, 1.5719945430755615, -0.3154538869857788, -0.2666130065917969, 0.10059875249862671, 0.44837984442710876, -1.2840744256973267, -0.5249879360198975, -2.0348551273345947, 1.4791754484176636, 0.6965809464454651, 0.6807883977890015, 1.9546538591384888, 0.49551400542259216, -0.7366247773170471, 0.08468315005302429, 1.6842330694198608, -0.1131688728928566, -0.17049570381641388, 0.2096303105354309, -0.524724543094635, 1.0886932611465454, 1.2804057598114014, 0.9695325493812561, -1.2553677558898926, 0.22453181445598602, -0.7183577418327332, -1.162827730178833, -0.3505862057209015, -0.6152214407920837, 0.035240162163972855, 0.9391888380050659, -0.5024179816246033, -0.8570224046707153, -0.10500619560480118, -0.3601078391075134, -1.3372923135757446, 0.48469680547714233, 1.2324435710906982, -0.5774707198143005, -0.05208548530936241, 1.0183746814727783, 0.8805720806121826, 1.3029508590698242, -0.13480599224567413, 0.35694369673728943, -0.5232839584350586, 0.21392236649990082, 0.323447585105896, 0.09435484558343887, -0.5912476181983948, -0.19675324857234955, -0.6767743825912476, -1.3391326665878296, 1.730018138885498, 1.3672404289245605, 0.8668863773345947, -1.598933458328247, 1.3185956478118896, -0.8988144397735596, 0.42196911573410034, 1.1243575811386108, 0.2353350967168808, 0.6694022417068481, 0.7450575828552246, -0.3562752902507782, 0.6649248003959656, 0.4939955472946167, -0.16069163382053375, -0.8549467325210571, 0.2949829697608948, -0.7510960102081299, -0.7501912713050842, 0.15176735818386078, 0.9420731663703918, 2.1924028396606445, 1.138974905014038, 0.213202103972435, -0.9691154360771179, 0.5627831816673279, 0.6357330679893494, 1.3275396823883057, 0.17442061007022858, 0.5062460899353027, 0.7937021851539612, -0.5222420692443848, -0.2879886031150818, 1.391077995300293, -1.211622953414917, 0.8699108362197876, 0.5576226115226746, 0.5254499912261963, 1.1430429220199585, 0.22077912092208862, -0.26938575506210327, -0.04296174272894859, -3.2828452587127686, 1.4624615907669067, 0.2822181284427643, -1.393326759338379, -0.5103927850723267, -0.2658778131008148, -0.7990198731422424, -0.35245904326438904, -0.5597496628761292, 0.1437891274690628, 0.17795853316783905, 1.5805286169052124, -0.567428708076477, -0.9999490976333618, -0.0459330677986145, 0.43627506494522095, 0.5523921251296997, 0.6099218130111694, -1.6863256692886353, -0.773528516292572, 1.3283696174621582, -0.5930705070495605, -0.7170834541320801, -0.5450853705406189, 1.0894560813903809, -0.5447902083396912, 0.5528460741043091, -0.6816022992134094, -1.0991425514221191, -1.467162013053894, -0.3738352358341217, 0.23314015567302704, -0.6841838359832764, 0.7326398491859436, -0.7638417482376099, -1.940493106842041, -0.06694496423006058, -0.949530303478241, -0.47505584359169006, 0.8981536626815796, -0.39449959993362427, -0.042570218443870544, 0.24681270122528076, -0.2313089221715927, 0.779303789138794, -0.6834734082221985, 0.6306633949279785, -1.5143117904663086, 0.14951184391975403, 1.3485310077667236, -0.9761226177215576, 1.42415452003479, 0.3227502107620239, -0.8939660787582397, 0.1911729872226715, 0.38260751962661743, -0.5923174619674683, 0.5015587210655212, 0.7280462384223938, 0.22953864932060242, 0.622362494468689, -0.718131959438324, -0.6926996111869812, -0.45512816309928894, -0.5422217845916748, 0.24447034299373627, -0.07526294887065887, -0.2905806303024292, -0.20575980842113495, 0.4449693560600281, -1.139457106590271, 0.4473664462566376, 1.348935842514038, -0.22822555899620056, 1.445015549659729, 1.7755670547485352, -1.1820894479751587, 0.974726140499115, -0.5478309392929077, 1.0040870904922485, 0.7285919189453125, -0.5054188966751099, 0.4377013146877289, -1.3557976484298706, 1.3225250244140625, -0.9365702271461487, 0.2048988938331604, 0.2447369396686554, 0.3465448319911957, 1.7469964027404785, 0.7339500188827515, -0.21877405047416687, -1.1546658277511597, 1.0617096424102783, -0.8405089974403381, 1.6188383102416992, 0.9602785706520081, 0.982157826423645, -0.11557348817586899, 0.5950218439102173, 0.11896684765815735, 0.7826387286186218, 0.6171013712882996, -1.207435131072998, 0.8455790877342224, -0.20117489993572235, 0.2559625208377838, 0.9071124196052551, -0.3646400272846222, 2.218958616256714, 1.9677937030792236, 0.29261815547943115, 0.7981438636779785, 0.16712597012519836, -0.9607975482940674, -0.7002912163734436, 0.6568084955215454, -0.15718317031860352, 0.9797689914703369, 0.7728380560874939, 2.062546968460083, -0.022697636857628822, 0.004758248571306467, 0.10872113704681396, -0.42002660036087036, -0.48399364948272705, -0.9740280508995056, -1.701900601387024, 0.6731685996055603, 0.3957298994064331, 0.9563857316970825, 0.883628249168396, 0.30930623412132263, -0.38915935158729553, 0.06490379571914673, -1.4642281532287598, 1.643478274345398, -2.4342856407165527, -0.8516033291816711, 0.3287408947944641, -0.19309455156326294, -0.48758459091186523, 0.8082110285758972, 0.32634276151657104, 0.14767296612262726, 0.12452660501003265, 0.2952522337436676, -1.1190948486328125, -2.065809965133667, 0.04311316832900047, 0.46765270829200745, -0.4431922435760498, 1.9911458492279053, 0.010289131663739681, -0.9968798756599426, -2.4418797492980957, 0.9535068273544312, -0.8945673108100891, -0.5682768225669861, 0.08522289991378784, 0.07960327714681625, -0.4658507704734802, -1.6228808164596558, -0.8455546498298645, 0.04489775374531746, -1.7547924518585205, 1.5002471208572388, -0.9918288588523865, 0.1816159039735794, 0.5214745402336121, 1.1069573163986206, 1.0140973329544067, 0.4997253715991974, 0.4786176383495331, -1.089986801147461, 0.5741587281227112, 1.2108124494552612, 0.29052218794822693, -0.4547940194606781, -0.5078652501106262, 1.53447425365448, -0.3642493188381195, 0.4485115706920624, -0.47262316942214966, 1.6275616884231567, -0.8711087703704834, 2.484813690185547, -0.7427711486816406, -0.4121549725532532, 1.8680830001831055, 0.7511463761329651, 2.8971941471099854, -0.3612841069698334, 0.22330044209957123, 0.5448467135429382, -1.8688530921936035, 0.6310024261474609, 1.4865775108337402, -0.20931856334209442, 1.0811423063278198, -0.48820000886917114, 1.0002312660217285, 0.917898416519165, 0.09179302304983139, 1.60834801197052, 0.6945153474807739, 0.4298081398010254, -2.3138608932495117, -1.372253179550171, 3.606788158416748, -0.5376790165901184, -0.09575314074754715, 0.7129588723182678, 0.2968851625919342, -0.4792904257774353, 1.8304693698883057, 0.42386871576309204, -1.0746403932571411, 1.0202391147613525, 0.40436869859695435, 0.20641209185123444, 0.541199803352356, 0.7475607395172119, -2.5992321968078613, -0.8101756572723389, 0.9445987343788147, -0.6351608037948608, 1.0719960927963257, -1.171025276184082, 0.7862223982810974, 1.0174639225006104, -0.7306787967681885, 0.7486233711242676, -0.6545944809913635, 0.5360523462295532, -0.7634142637252808, 0.18806064128875732, 0.010630081407725811, -1.868425965309143, -0.9706371426582336, 1.0016769170761108, 0.6779858469963074, 0.21083347499370575, 0.855739951133728, 0.2346644401550293, -0.42294636368751526, -0.15136180818080902, -1.1305028200149536, 0.3452543020248413, -0.23200272023677826, -0.29785293340682983, 0.5827766060829163, -0.8862678408622742, 0.21411116421222687, -0.5015125274658203, -0.7407466769218445, -0.1445252150297165, 1.7847974300384521, -0.8847344517707825, -1.8239840269088745, 0.2108316868543625, 0.4712323248386383, 0.07987338304519653, 0.09965912252664566, 1.779574990272522, 0.3453032970428467, -0.024639708921313286, -1.1642519235610962, -0.48020875453948975, 0.4695281386375427, -0.9415580034255981, -0.27126461267471313, -1.3015351295471191, 0.33215731382369995, -0.6512760519981384, 0.25452694296836853, 0.3267629146575928, 0.9539648294448853, -1.8954145908355713, -0.860403835773468, -0.7037896513938904, -1.1108070611953735, 0.07056733220815659, 0.780501663684845, -0.9836622476577759, 1.0807760953903198, -0.7926761507987976, 0.24695098400115967, -0.9987208843231201, 0.1474546194076538, -0.4677424430847168, 0.9986534714698792, -1.0140904188156128, -0.3799189329147339, 1.939497709274292, 0.8457583785057068, 1.0771514177322388, -0.41569027304649353, 0.5776995420455933, -0.6745654940605164, -1.329312801361084, 0.7681643962860107, 1.7388406991958618, 0.5333235263824463, 1.0362147092819214, 0.26471376419067383, -0.5120329260826111, -0.23486275970935822, 0.5859653353691101, 0.5351918935775757, -0.7142292261123657, -0.3429486155509949, 1.9946390390396118, 0.5465543270111084, -0.600027322769165, -1.138715386390686, -0.01737821474671364, 0.6939314603805542, -0.6416419148445129, -0.2905985713005066, -1.3394501209259033, 0.03964817151427269, 0.009394300170242786, -0.02435658499598503, -0.8650808930397034, -0.3671899139881134, 1.514733910560608, -1.10130774974823, 0.840545117855072, 0.34748899936676025, -1.4476933479309082, -1.9731065034866333, 0.8150279521942139, -1.5320932865142822, 1.1263235807418823, -1.6296772956848145, -0.29270586371421814, 0.19865180552005768, -0.16681820154190063, -2.5635199546813965, -1.7280138731002808, -0.24417825043201447, -0.592051088809967, -1.1851366758346558, 0.30844539403915405, -0.9348245859146118, -0.5124484896659851, -1.0345523357391357, 0.7633978128433228, -1.0415573120117188, -0.6064008474349976, 1.5347546339035034, -0.06438227742910385, 1.3236114978790283, 0.14135029911994934, -0.15711969137191772, 0.28472191095352173, 0.6748352646827698, -1.305057406425476, -0.612438440322876, 0.6319179534912109, -0.09685026854276657, -0.26514503359794617, -0.564044177532196, 1.3305153846740723, 0.4286844730377197, 0.4670875370502472, 1.311302900314331, 0.8359470367431641, 0.10967957973480225, -1.116348147392273, 0.06954440474510193, -1.1264123916625977, -0.3166500926017761, 1.5498791933059692, 1.254468560218811, 0.7240622639656067, -1.2296929359436035, -0.09602522850036621, -0.43130284547805786, -0.3399648070335388, 0.73615962266922, 1.7393133640289307, 0.9012653827667236, 2.145246744155884, -0.6081163287162781, -1.0331169366836548, 0.3029475808143616, -1.4800456762313843, 0.06944135576486588, 0.44550859928131104, -0.7286814451217651, -2.1260693073272705, -0.1766173243522644, -0.01581614278256893, -1.5942455530166626, -0.48734551668167114, -1.1247034072875977, -0.3277031481266022, 0.3186851739883423, 0.6338120102882385, -0.41636383533477783, 1.1627788543701172, -1.0865947008132935, 1.3087828159332275, 0.055145785212516785, 0.6594396829605103, 0.2698829770088196, 1.1684072017669678, -1.18161141872406, -0.8909961581230164, 0.7545350193977356, 0.8985126614570618, -1.7765275239944458, 0.9949223399162292, 0.03616967424750328, 0.9909471273422241, 0.045139130204916, -0.39841750264167786, 0.1578138768672943, 0.43430688977241516, 2.4144718647003174, 0.2341228872537613, 0.9495276808738708, -1.7670154571533203, -0.15742357075214386, 0.3237144947052002, 1.3392431735992432, 0.8641170263290405, -0.8131662011146545, 0.48077526688575745, 0.13883662223815918, 0.3166812062263489, -0.6950019598007202, 1.3207610845565796, -1.0371532440185547, 0.2583369314670563, 0.42956337332725525, 0.45650714635849, 0.3680213391780853, -0.6675969362258911, -1.0500754117965698, -0.18331614136695862, -0.43471989035606384, -0.1397097259759903, -0.7547121644020081, 0.9361124634742737, -0.645377516746521, -0.9052855968475342, 0.4416358470916748, 0.2314310222864151, -0.053589265793561935, -1.0945320129394531, -1.537693977355957, -1.1610838174819946, 0.6282760500907898, 0.5964531302452087, -0.24042432010173798, -0.7934266328811646, -0.8988251686096191, 0.48971930146217346, 0.9150946736335754, -1.4059152603149414, -0.6620596051216125, 0.6274148225784302, 0.8562848567962646, -0.9399783611297607, 0.07104489207267761, 2.0815694332122803, 0.9349910616874695, 0.6913297176361084, -0.667496383190155, 0.24390809237957, 0.6875866055488586, -0.7039819955825806, 1.0399558544158936, -0.4614420533180237, 2.2355802059173584, -0.33629515767097473, -1.4409488439559937, -0.5823167562484741, -2.159036636352539, 0.8609652519226074, 0.6011331677436829, 0.7522094249725342, 1.4213701486587524, -0.18021231889724731, 2.030766010284424, -0.8697052001953125, -2.771252155303955, 0.5470094084739685, 1.578480839729309, -0.6565625667572021, 0.27899470925331116, 0.7366169691085815, 1.4754351377487183, 1.13958740234375, 0.7751049399375916, 0.36880356073379517, -2.8312418460845947, -0.5237695574760437, 0.7139279246330261, 0.7537423372268677, 1.1089028120040894, 1.2098157405853271, -1.122737169265747, 0.30722320079803467, -1.3797273635864258, -0.6963518857955933, -1.2432198524475098, 1.6037039756774902, 0.8315535187721252, 1.5397686958312988, 0.5408092737197876, 0.27516257762908936, -0.07986647635698318, -0.1755565106868744, -0.39081668853759766, -0.09601263701915741, 0.22608982026576996, -0.7033506631851196, -0.6109458208084106, -0.23378024995326996, -0.14882442355155945, 1.7322083711624146, 0.08758877217769623, 0.31282171607017517, 0.6030116677284241, -0.0284931231290102, -0.4775375723838806, -0.6632246971130371, 0.04091769456863403, 0.7829862236976624, -0.056228239089250565, -0.31595832109451294, -0.563387930393219, 1.4924566745758057, -0.7500067949295044, -0.1967058777809143, 0.05844912678003311, 0.8379420042037964, -1.6878399848937988, 0.013698692433536053, -0.2252136468887329, -0.6722515225410461, 0.3218086063861847, -0.01710636168718338, -0.6350552439689636, -1.234491229057312, -0.1353486329317093, -0.3361057639122009, 1.0872039794921875, 1.0148115158081055, 1.0885848999023438, 0.5899476408958435, -0.4183313846588135, 0.7221217155456543, 0.22491082549095154, 0.6058638691902161, -1.508999228477478, 1.6830281019210815, 0.359383225440979, -1.2770001888275146, 1.7181851863861084, -0.47535398602485657, -1.913623332977295, -1.0382108688354492, 0.24542522430419922, 1.6815849542617798, -0.44752296805381775, -0.9713774919509888, 0.6690104603767395, 0.43535667657852173, 1.9595680236816406, -0.04815457761287689, -0.38854077458381653, 0.03283022344112396, 1.099921464920044, -0.44969144463539124, -0.7356405258178711, 2.171739101409912, -0.9023768305778503, 1.7582389116287231, -0.9625580310821533, 0.9260731339454651, -0.842738151550293, -0.10193004459142685, 0.38438066840171814, -0.21839898824691772, -0.15815110504627228, -0.27451133728027344, 1.0780423879623413, -1.162536859512329, -1.194295883178711, 0.06647185236215591, 0.6343691349029541, 0.1281975656747818, -0.10491757839918137, 0.4769043028354645, -0.6296223998069763, 0.826262891292572, 0.21978650987148285, -0.40278899669647217, -0.5651345252990723, 1.309322476387024, -1.2087615728378296, -0.6059499979019165, 0.04666651040315628, 0.2323945015668869, 1.8549801111221313, 0.24930675327777863, -1.0950173139572144, 0.6518839001655579, -0.29728594422340393, 0.17924311757087708, -0.28526341915130615, 0.1834438443183899, 0.19704493880271912, 1.6879336833953857, -0.3710198700428009, -0.8684569001197815, 2.854525089263916, 0.9536356329917908, -1.0107638835906982, 0.6742225885391235, -0.33205679059028625, -1.2134945392608643, 1.9409232139587402, -0.7003524303436279, -0.4517463743686676, -0.8091560006141663, 1.2893401384353638, 1.2080538272857666, -1.1897202730178833, -1.3345707654953003, -0.7201593518257141, -0.4302051365375519, 0.5372714400291443, 1.2971996068954468, -0.3109874725341797, -0.5036001801490784, -0.2463148534297943, 0.34368184208869934, 1.8186309337615967, 0.19414907693862915, 0.10448868572711945, 0.8345863819122314, 0.8318560719490051, 0.5678048133850098, 0.7801096439361572, 1.0810273885726929, -2.7266385555267334, 0.5185366868972778, 1.4653998613357544, 0.27924007177352905, -0.857719361782074, 1.0977251529693604, 1.5769985914230347, -1.5340849161148071, -1.3706718683242798, -0.6169025897979736, 1.302738070487976, 0.8287379741668701, 0.41406679153442383, -0.2801620364189148, 1.7131634950637817, 0.10150284320116043, -0.3649290204048157, -0.3727911412715912, 1.7546031475067139, 0.4267353415489197, 0.9979892373085022, -1.1563432216644287, -0.8266186714172363, -0.47290652990341187, 0.6734124422073364, 1.8154191970825195, 0.9243177771568298], [-0.3144620656967163, -0.11074449867010117, -0.05005916208028793, 0.09667922556400299, -1.4466499090194702, -1.5035412311553955, 0.22224143147468567, 0.892228364944458, 1.0507729053497314, 0.2844332754611969, 0.09179145097732544, 0.859938383102417, 0.6599220037460327, 0.14399294555187225, 1.6711307764053345, -0.6128174066543579, 0.31946906447410583, -0.6208064556121826, -0.8471251130104065, 1.2457908391952515, 0.8316406011581421, 0.9745210409164429, 1.490120768547058, -1.5026999711990356, 0.6005042195320129, 0.2663113474845886, -1.765551209449768, 1.29255211353302, 1.4153493642807007, -0.2387869954109192, 0.7164583206176758, -0.044666897505521774, -1.3837398290634155, -0.11894400417804718, -1.0525654554367065, -0.03357896953821182, 0.16459155082702637, 0.5691160559654236, 0.5813959836959839, 0.5979930758476257, -0.2183721512556076, -0.769261360168457, -0.1345720887184143, -0.5449158549308777, 1.364808201789856, -0.33679986000061035, -0.7190314531326294, 2.326298713684082, -2.2801759243011475, 2.347095251083374, 0.33079642057418823, 0.402558833360672, -0.555533766746521, -0.029402317479252815, 1.159603476524353, 0.7321853637695312, -1.8019323348999023, 0.14468781650066376, 0.45474156737327576, 0.8253648281097412, 1.3935250043869019, -0.43248769640922546, 1.706845760345459, 0.6416661143302917, -1.378249168395996, 0.9130879044532776, 1.669822096824646, -1.2061352729797363, 0.9252920746803284, 0.1999371498823166, 0.8189994096755981, -1.2819596529006958, 0.3057873845100403, 0.5273152589797974, -0.6552244424819946, 0.5381274223327637, -0.5932444930076599, -0.4831683933734894, -0.262128084897995, -1.623058557510376, 0.9689406156539917, -0.17258477210998535, 0.5199487805366516, 0.07884526997804642, -0.8993653059005737, 1.2781257629394531, -1.249510407447815, 1.1506621837615967, 0.16301299631595612, 2.2476863861083984, 0.4037478268146515, -0.616173267364502, 0.23371522128582, 0.8302812576293945, 0.7098626494407654, -0.23618170619010925, -0.03322070837020874, 0.33855462074279785, -1.3940869569778442, -0.8630646467208862, -0.41142818331718445, 0.4218955338001251, -0.2680763602256775, 0.7742668986320496, 0.9540493488311768, 0.26312175393104553, -0.3915378451347351, -0.42120516300201416, 0.606026291847229, -1.4757354259490967, 0.44418081641197205, 0.645447850227356, 0.10444268584251404, 0.16194592416286469, -0.3057214617729187, -0.49292701482772827, 0.7822498679161072, -0.5501774549484253, -2.095353126525879, 1.4675726890563965, 0.402213990688324, -0.721380889415741, 0.9011304378509521, 0.921841025352478, 0.8017486929893494, -0.2601596713066101, 1.63288414478302, -1.0604451894760132, 2.318533420562744, 0.3980247378349304, -0.6697694063186646, -0.5826820731163025, -0.33166319131851196, -1.1284737586975098, -0.605122447013855, 0.0018580458126962185, -0.12714852392673492, 0.33083975315093994, -2.1847925186157227, 0.005854073446244001, -1.0411038398742676, -1.4158562421798706, -0.836663007736206, -0.33956873416900635, 0.3095118999481201, 0.25357916951179504, -0.5099181532859802, -2.2073652744293213, 1.4363651275634766, 0.14465923607349396, -2.692643642425537, -0.18086469173431396, 0.8843377232551575, 0.29753291606903076, -1.058725118637085, -2.497809886932373, 2.139328956604004, 2.044199228286743, 0.65143221616745, -1.088340401649475, 0.5845029354095459, -0.07066252827644348, 0.0658862367272377, 0.6566290855407715, -2.6653590202331543, -0.008307725191116333, -0.5646328330039978, -0.8003742098808289, 2.0721163749694824, -2.1484017372131348, 0.5219597220420837, 1.6056005954742432, 0.8314189314842224, 0.5443966388702393, 0.46999257802963257, -0.5336415767669678, -1.1771854162216187, 0.3139786422252655, 0.18912366032600403, -1.326341986656189, -0.01022084429860115, -0.44251447916030884, -0.6835923790931702, -0.9627028703689575, -1.1399731636047363, 0.5961563587188721, 0.28865277767181396, 0.3368057310581207, -1.95264732837677, 1.4442956447601318, -1.7145602703094482, 0.07220316678285599, -0.34797239303588867, 0.600210428237915, -0.3007417321205139, 0.5388431549072266, -1.2701560258865356, -0.311181902885437, 0.024500150233507156, 0.590682327747345, 1.1121424436569214, -1.0361038446426392, -0.06212856248021126, 1.0706019401550293, -0.8638575673103333, -0.03369498625397682, 0.5974263548851013, -0.49890047311782837, -0.2591518461704254, 0.3489440977573395, -0.22307617962360382, -0.6085718274116516, -0.33567431569099426, -1.4258689880371094, 0.4642818868160248, -0.14064277708530426, 0.2873547077178955, 0.6613665223121643, -1.7961652278900146, -0.9994609951972961, 1.9838829040527344, -0.18630535900592804, -0.10694075375795364, 0.8205209374427795, 1.5240802764892578, -1.3102573156356812, -1.3269559144973755, -0.053941383957862854, 1.03074049949646, -0.2075565755367279, -0.40115174651145935, -0.05123278498649597, 3.040832281112671, -0.784741997718811, -0.9851981401443481, -0.6993734240531921, 0.18095864355564117, 0.427792489528656, 0.6004691123962402, 0.5093845725059509, -0.828870952129364, 0.11825621128082275, 1.0547473430633545, 0.4894074499607086, -0.017583778128027916, -0.5621120929718018, 0.15387682616710663, 0.8908451795578003, -1.467860221862793, -1.0044991970062256, 2.2502410411834717, 1.6148465871810913, 1.5728116035461426, -0.26979148387908936, 0.1414412409067154, -0.3829856514930725, 0.29434117674827576, 0.6539993286132812, 0.24011239409446716, -2.5505285263061523, 0.6254079937934875, 0.8488034009933472, 0.24874582886695862, 0.4571853578090668, 0.6059930920600891, 0.2658539414405823, 1.2413585186004639, -0.8887068033218384, 0.39998486638069153, -1.0325278043746948, -0.9292709827423096, -0.44555604457855225, -0.2357485592365265, -0.41916796565055847, 1.8482112884521484, -0.6993504762649536, -0.3781573176383972, -0.8474136590957642, -2.304685115814209, 1.120405912399292, 0.4437016546726227, -0.7919941544532776, 0.02416258677840233, 1.8022900819778442, 1.1142834424972534, 0.11058962345123291, 0.3241421580314636, -0.3507889211177826, 0.6109418869018555, 0.8129015564918518, -1.3391753435134888, 1.308388590812683, 0.557238757610321, 0.4547230005264282, -1.2109310626983643, -0.4776788651943207, 0.9667595028877258, 1.0009955167770386, -0.8673049211502075, -0.7868393063545227, 0.7184250354766846, -0.7239354848861694, 0.8490511775016785, -1.0076431035995483, 0.9066796898841858, 0.7364471554756165, 1.299932599067688, 2.4433536529541016, 0.6332604289054871, 0.9455229640007019, -0.3524593710899353, 0.3248918652534485, 2.4671390056610107, -0.21954858303070068, 1.4592925310134888, 0.6645953059196472, -0.020169779658317566, -0.3881118595600128, -0.7113634943962097, -0.6732490658760071, -2.9052822589874268, 1.5615875720977783, -0.6726447939872742, -0.7462373375892639, -1.32896888256073, -0.22313585877418518, -1.175895094871521, 0.4882895052433014, -0.33886244893074036, -0.5571339130401611, 0.8806952834129333, 0.357713907957077, 0.950916051864624, -1.199699878692627, -0.3793746531009674, -0.12829884886741638, 0.6310628056526184, -1.0317260026931763, 1.2934786081314087, -0.0727483481168747, 0.31585702300071716, -0.030232252553105354, -1.3457807302474976, 0.3011355400085449, -2.0348000526428223, -0.020780202001333237, -0.5783337950706482, -0.4768734872341156, -0.10993537306785583, 1.6742255687713623, -1.2332264184951782, 0.2784667909145355, -1.6068072319030762, 2.2203922271728516, 0.9927349090576172, -0.6930404901504517, 1.1891779899597168, 1.206257700920105, -2.0606818199157715, -1.277306079864502, 0.5687341094017029, -0.24516460299491882, -0.36108317971229553, -0.30180811882019043, -0.5888307094573975, -0.8751338720321655, -1.035116195678711, -0.9387000203132629, 0.6136232614517212, -0.6812735795974731, -0.48645395040512085, 1.4369045495986938, 0.9378679394721985, -0.4973739683628082, -0.08964184671640396, -0.3744922876358032, -0.7773003578186035, 1.049190640449524, 1.7948768138885498, 1.1758379936218262, -1.1584769487380981, 0.7604435086250305, -0.1495068520307541, -0.9529105424880981, 0.12421514093875885, -0.01976066641509533, 0.6984222531318665, -0.9165143370628357, -1.2074021100997925, 0.40011802315711975, -0.32008686661720276, -1.2540146112442017, -0.8640958666801453, 1.0126547813415527, 0.5786653161048889, 1.1713261604309082, -0.036282919347286224, 0.017361698672175407, 0.2553067207336426, 0.6485915184020996, 0.686143159866333, -1.051382064819336, 0.5041643381118774, 0.27221640944480896, 0.5823730826377869, 1.6407029628753662, 0.654852569103241, -0.10218357294797897, 0.022126594558358192, 1.6285414695739746, 1.4427871704101562, -0.226797953248024, -1.2086430788040161, 0.4074476361274719, -1.0346776247024536, -0.13297347724437714, -0.7972031831741333, 1.331191062927246, 0.4240521192550659, -1.887566089630127, 1.4666869640350342, -1.273993730545044, 0.0913950651884079, 0.18986976146697998, -0.6055675745010376, -1.2643200159072876, -0.4405323266983032, 1.59494948387146, -0.5569151639938354, 0.2050752341747284, 1.5284991264343262, 0.20644669234752655, -0.5541802048683167, -0.4540276825428009, -1.1759637594223022, -0.26141807436943054, -0.44584015011787415, -1.673392653465271, -0.07337073981761932, -0.18521347641944885, 0.5015055537223816, -0.34844115376472473, 2.2640693187713623, -0.29787683486938477, 0.483323335647583, 1.1792110204696655, -1.2525190114974976, -0.9985329508781433, 0.6930630207061768, 0.27662909030914307, -1.9988114833831787, 0.41331297159194946, -0.3388388156890869, -1.3734480142593384, -0.6209784150123596, 0.9099702835083008, 0.9819974303245544, -1.8335838317871094, -1.2135604619979858, -0.987765908241272, -0.31641167402267456, -1.5940004587173462, 0.04151886701583862, 0.805712103843689, 0.8463274836540222, 1.9631060361862183, 1.7652629613876343, 1.3563231229782104, 1.375230073928833, 0.3963180482387543, 0.06803618371486664, -0.8072872161865234, 0.6404628753662109, 1.3349612951278687, 1.135424256324768, 1.0298837423324585, 0.870396077632904, 0.16699287295341492, -1.5924726724624634, 0.7232286334037781, -1.4317961931228638, 1.5191268920898438, -1.794191837310791, 0.0035878540948033333, -0.6446171402931213, 0.018160343170166016, 0.34248411655426025, -0.8316567540168762, -1.3385323286056519, -0.09285645186901093, -0.3676326870918274, 2.1013267040252686, 0.8372389674186707, 0.7168049812316895, 0.10557330399751663, -0.7919670939445496, 0.4949493110179901, -0.12273901700973511, -0.07642969489097595, 0.6049434542655945, -0.033193688839673996, 0.03176829218864441, 0.12692058086395264, 0.13263316452503204, -0.22433505952358246, 0.27957168221473694, -0.702458918094635, -1.3004859685897827, 1.230764627456665, -1.5228701829910278, -1.4965842962265015, -0.4404763877391815, -0.0785328820347786, 0.9183701276779175, 1.7914941310882568, 0.29682132601737976, 0.3222809135913849, -1.1123591661453247, -1.0948675870895386, -1.3325526714324951, -0.006165010388940573, -0.7313407063484192, 1.207261562347412, 1.1432386636734009, -1.5376955270767212, 0.5207767486572266, -0.39398667216300964, -0.9402855634689331, 0.3820820152759552, -1.3381130695343018, 1.5639325380325317, 0.10596264898777008, -0.14865832030773163, 1.7100728750228882, -0.9053528904914856, 0.8050239086151123, -0.03733101487159729, 0.23626624047756195, -0.5131087303161621, 0.5172685384750366, -0.08940979093313217, 0.3708137273788452, -0.6024667620658875, 0.009425722993910313, 1.184184193611145, -1.4594734907150269, -0.7745610475540161, -1.1327550411224365, -0.2580014169216156, -0.4030047655105591, 1.2268399000167847, 1.573239803314209, 1.1207523345947266, 0.9930064082145691, -0.3891831040382385, -1.3176281452178955, -0.14190010726451874, -0.46345770359039307, 0.8178620338439941, 2.1347620487213135, 1.1892997026443481, 0.5164188146591187, 1.5008211135864258, -1.1002745628356934, 0.3256003260612488, -2.0325026512145996, -0.7327788472175598, 0.4176379144191742, 1.2652369737625122, -0.2773878276348114, -1.8594111204147339, 0.6734375357627869, -2.7163209915161133, 0.2821459174156189, -1.149641990661621, 0.0557061992585659, 0.33288639783859253, -0.25037190318107605, -2.055244207382202, 0.16347326338291168, 0.1903291642665863, -0.15988700091838837, 0.12790971994400024, 0.4875487983226776, 0.462399959564209, -0.21685078740119934, 0.11335237324237823, -0.8822938799858093, 0.6750748157501221, 0.28935185074806213, 0.10471313446760178, -1.9668099880218506, 0.8504891395568848, -0.5167500972747803, -0.10655945539474487, 1.0479235649108887, 0.6809125542640686, 0.6918773055076599, -1.8795584440231323, -0.8445911407470703, 0.8971142768859863, 0.44407954812049866, 0.6593151688575745, -0.49879705905914307, -0.30989959836006165, -1.1434773206710815, 0.19907128810882568, 0.04114840179681778, 0.8761739730834961, 2.981851100921631, -0.8665277361869812, 1.5579559803009033, 1.1089863777160645, -0.23080457746982574, -0.5563897490501404, -0.8373737335205078, -1.4162356853485107, 0.9020274877548218, 0.12695196270942688, 0.9240544438362122, -1.8520162105560303, 1.956717848777771, 0.6139277815818787, -0.5560527443885803, 0.7756260633468628, -2.36295747756958, 1.1546517610549927, -0.7916112542152405, -1.8671002388000488, 0.3225395381450653, 0.06433922052383423, -0.41563332080841064, 0.3909490704536438, 2.635023593902588, 0.4475629925727844, 0.4896065592765808, -0.024488743394613266, -1.3725335597991943, 0.10100415349006653, 1.5649501085281372, -1.4735841751098633, -1.7783879041671753, -0.2885659337043762, 1.2295151948928833, 0.5385063886642456, -0.22914670407772064, -0.07584378123283386, 1.3296500444412231, 0.6037826538085938, 0.07382190227508545, 0.18094483017921448, 0.12015897035598755, 0.29266008734703064, 0.8963314294815063, 0.3001321852207184, -0.984361469745636, 0.5012605786323547, -0.16280993819236755, 0.6551087498664856, 0.5520887970924377, -0.8253282308578491, -0.9057464003562927, 0.8118153214454651, 1.0638729333877563, -1.1674091815948486, 1.479836106300354, -0.14592896401882172, 0.15456554293632507, -0.11627651005983353, -2.597038984298706, 1.3462755680084229, 0.10318360477685928, 1.507057547569275, -0.1785147339105606, 0.8526571393013, -0.36320051550865173, -0.46472832560539246, -0.6363219022750854, -0.4521157145500183, -1.0289356708526611, -1.0130021572113037, -1.317873477935791, -3.117804765701294, -0.3661929666996002, 0.4691374599933624, 0.9634059071540833, 0.11236967146396637, -0.39602476358413696, -1.4338726997375488, -0.8980652689933777, -2.6358399391174316, 2.1354846954345703, 0.8694952726364136, 0.17551293969154358, 1.0062546730041504, 0.29666078090667725, 0.4011431932449341, -0.3178393244743347, -0.34884941577911377, 0.3930906057357788, -0.5577263832092285, 0.06670914590358734, -0.2748408317565918, -0.15616333484649658, 0.5741046667098999, -0.3288937509059906, 0.3381076157093048, -0.020988643169403076, 1.3316161632537842, 1.1435707807540894, 1.35617196559906, 1.0051863193511963, -0.28518179059028625, -1.029158592224121, -0.6477407813072205, 1.5117053985595703, 0.03348301351070404, -0.5796976685523987, 1.1321240663528442, 0.30546438694000244, 1.2323567867279053, -0.8444938659667969, -0.11512032896280289, 0.8114265203475952, 0.7396793365478516, -0.8556168079376221, 1.224013328552246, -1.2386137247085571, 0.0068214102648198605, -0.03477029874920845, 0.8170299530029297, -1.0542582273483276, -0.2464253306388855, -0.8797721266746521, 0.904232382774353, 0.12690727412700653, -0.3221265971660614, 0.6137401461601257, -1.4238123893737793, 0.21184982359409332, -0.5485182404518127, 0.8188320398330688, 0.05766994506120682, -0.40115901827812195, -1.6237671375274658, 0.432466596364975, -0.7474285960197449, 0.20378729701042175, -0.005668362602591515, -0.06412605196237564, -0.5012693405151367, -0.44155335426330566, 1.5966445207595825, -0.10696889460086823, -0.5741031765937805, 1.3729417324066162, 1.5167055130004883, -0.5993144512176514, 0.3836984932422638, 1.2820597887039185, 0.6511280536651611, 0.5820310711860657, 1.8425155878067017, 1.3360543251037598, -0.2969779670238495, 0.39815396070480347, 0.15038242936134338, 1.5887718200683594, -1.068068265914917, 0.49950262904167175, 0.01497520413249731, -0.5999270081520081, 0.3228152394294739, -0.5854124426841736, -0.41967493295669556, -1.210057020187378, -1.0441089868545532, -0.5390575528144836, 0.7094579935073853, -0.8470211029052734, 0.13177965581417084, 0.4790544807910919, 0.6215515732765198, 0.9224007725715637, 0.39761102199554443, -0.8982582688331604, -1.118137001991272, -0.4059085547924042, -0.3878368139266968, 1.4720449447631836, 0.516560971736908, 2.0045573711395264, -1.099530577659607, -2.4304537773132324, 0.9968616962432861, -1.3069754838943481, 0.3340955078601837, 1.1552565097808838, -1.7957518100738525, -1.8138939142227173, 1.376663327217102, 0.8581051230430603, 0.40482625365257263, -2.5569207668304443, 0.2974741756916046, 0.13992665708065033, -1.8260979652404785, 0.9962078332901001, 0.291300505399704, 0.6605521440505981, 0.73197340965271, -0.6208285689353943, -2.2191195487976074, 0.13871189951896667, 1.2992957830429077, -1.0987811088562012, -1.144668459892273, -0.33977800607681274, 0.8464644551277161, -0.5400875806808472, 0.5459656119346619, -0.30935347080230713, 1.6434204578399658, 1.4198317527770996, -1.036068320274353, 0.12393966317176819, 0.6515800952911377, -0.9300491213798523, -0.130059152841568, -0.5934443473815918, -0.11367926001548767, 0.7208960056304932, 0.7257418632507324, 1.0711348056793213, -1.7978283166885376, -1.5923796892166138, -0.4354270398616791, -0.0991167202591896, 2.173841953277588, -1.3173656463623047, 1.1094263792037964, 2.603864908218384, -0.5508866906166077, -0.451485812664032, -0.22160467505455017, 0.10935879498720169, -1.4761444330215454, -0.6220433712005615, -1.1728249788284302, -0.3710254728794098, 1.4034907817840576, 2.262007474899292, 1.5300867557525635, 0.44291937351226807, -0.8942831754684448, 0.4752073884010315, 0.6451483368873596, -0.2008821964263916, -0.06329231709241867, -0.24777404963970184, 0.45643317699432373, 0.7261529564857483, 1.3279114961624146, -1.8583794832229614, -0.7078992128372192, 0.4040238559246063, -0.09856000542640686, 1.5866546630859375, 0.5575068593025208, -0.11444205045700073, 0.40999290347099304, 0.7488154172897339, -0.7123419046401978, 0.4618765711784363, 0.07187351584434509, 0.47529441118240356, 2.538741111755371, 0.6112651228904724, -0.03590257838368416, 1.054105281829834, -0.38857588171958923, 2.093873977661133, 0.5505701303482056, -0.10389682650566101, 0.8362458348274231, 0.5146324634552002, 0.015054206363856792, -0.38063403964042664, 1.6231260299682617, 0.08573468029499054, -1.0530604124069214, 0.7321494221687317, 2.0870134830474854, 0.43167969584465027, 0.7985877990722656, -0.10199061036109924, 0.06704198569059372, -0.03145236149430275, -1.6874083280563354, 0.032662760466337204, 0.974091112613678, 1.2710163593292236, 0.29344186186790466, 1.225977897644043, 0.7814635634422302, -0.8717743158340454, 0.1176256462931633, -0.6008559465408325, -0.004586364608258009, 0.10758472979068756, 0.7445778250694275, 1.8437824249267578, -0.01643197238445282, -1.0890721082687378, 0.2970200479030609, -0.30111628770828247, 0.2748596966266632, 0.5227178931236267, -0.449588418006897, -2.4177894592285156, 0.85072261095047, 0.44401976466178894, -0.05179116874933243, 2.4870643615722656, -0.2406594455242157, 0.3623339831829071, 0.1376846581697464, 0.981757640838623, 1.4578214883804321, -0.5515084266662598, 1.899375557899475, -0.7054533958435059, 0.6688036322593689, -1.3057368993759155, -0.8162412047386169, -1.631795048713684, 0.3474321663379669, -0.46670758724212646, -0.18934723734855652, -0.9605388641357422, -0.1942828893661499, -1.0688915252685547, 1.198102593421936, 1.5955069065093994, -1.2658631801605225, -1.3451069593429565, 0.05778555944561958, 0.3304738998413086, 0.25787150859832764, 0.28883394598960876, 0.23806743323802948, -1.3500910997390747, -0.2247970998287201, 0.14871661365032196, 0.5315477848052979, -1.6954525709152222, -0.2339988797903061, 0.29089176654815674, -0.9358857274055481, 1.5992283821105957, -1.3898905515670776, 0.8782659769058228, 0.8697552680969238, 0.2611146867275238, 0.25218427181243896, -0.07475273311138153, -1.5127689838409424, -0.6318181753158569, 1.4832786321640015, 1.2483724355697632, -0.24293354153633118, 2.022648811340332, -0.09663816541433334, -0.2090756744146347, 1.4279521703720093, 1.5107942819595337, -0.17555254697799683, -2.2398152351379395, -0.4390352964401245, -0.28677573800086975, -0.6794306635856628, -0.9553229212760925, 1.0034464597702026, 1.3859407901763916, -0.383379727602005, -0.3372025489807129, -0.9140667915344238, 0.16821981966495514, 0.5190554857254028, -0.39016208052635193, -0.9865539073944092, 0.3490731120109558, -0.6360135078430176, -1.5732334852218628, -0.8670077919960022, 0.07414467632770538, -0.17553414404392242, -1.8707166910171509, -0.5727325081825256, -0.06376972049474716, -0.2986610233783722, -1.7621961832046509, 0.18105848133563995, 1.095880150794983, -0.4115864634513855, -0.8123403191566467, 0.9024320244789124, 0.7164577841758728, 0.7605748176574707, -0.339933842420578, -2.4475369453430176, 2.116060256958008, -0.8262403607368469, 1.4878498315811157, -0.0588570162653923, 0.9616814255714417, -1.0751644372940063, -0.1437404304742813, -0.15545473992824554, -0.24633021652698517, -0.5347835421562195, -1.336214303970337, -1.552861213684082, -0.38067692518234253], [-0.20019587874412537, 0.9808436632156372, 0.7875006794929504, 0.6025252938270569, 2.0193281173706055, 1.9098577499389648, 0.16095811128616333, 0.8996699452400208, -1.6946097612380981, -0.5338794589042664, -1.1462739706039429, -1.0382367372512817, -0.130500927567482, -1.3268473148345947, -0.09066816419363022, -0.9609965085983276, 1.9638338088989258, -1.4511343240737915, 0.3612620234489441, 0.1447906792163849, -0.9953114986419678, 1.0257582664489746, 1.128246545791626, 0.5454291701316833, 0.709511935710907, 1.2508461475372314, -0.007811331190168858, 0.13965992629528046, 0.2586512267589569, 1.042022943496704, -1.2567747831344604, -1.4262436628341675, -0.5880748629570007, 1.6518174409866333, -1.5762745141983032, -0.8261064887046814, -0.6236554384231567, 2.1984221935272217, -0.6776314377784729, 0.012887302786111832, 0.5084474682807922, 2.209784507751465, 0.4383943974971771, -0.757489025592804, 0.3730299770832062, -0.27146297693252563, 1.1737908124923706, -0.8889815807342529, 1.9204105138778687, 0.35033130645751953, 2.3723785877227783, -0.817852795124054, -0.1845959573984146, 0.684109628200531, 0.5568137168884277, 1.1993415355682373, -0.559492826461792, -0.5411209464073181, -0.17647753655910492, 0.09578592330217361, -1.4637986421585083, 0.5047339200973511, 0.1920308917760849, -0.36402639746665955, 2.2102913856506348, -0.2047620266675949, -0.7001640796661377, 1.064673900604248, -0.8490992188453674, 1.0641900300979614, 0.29504239559173584, 0.9493705630302429, 0.0642055943608284, -0.533966600894928, 0.027271850034594536, 0.14206130802631378, -0.16267335414886475, -1.1995161771774292, -0.7597226500511169, -0.26605525612831116, -0.7056682705879211, -1.5467318296432495, -2.283902406692505, 1.2575005292892456, 0.721412718296051, -0.32519927620887756, 0.5048073530197144, 0.13207364082336426, -0.701036810874939, -0.6720735430717468, 1.4135397672653198, -1.056923747062683, -2.4234564304351807, -0.29919543862342834, -0.808143675327301, -0.7212663888931274, -0.5359717607498169, -0.8918930888175964, -0.13515938818454742, -0.29747384786605835, 1.9570589065551758, -1.8358268737792969, 0.3990061581134796, -0.7501853704452515, -0.18718136847019196, 0.03198837861418724, 2.1712069511413574, -1.3692179918289185, -0.026028765365481377, -1.3555574417114258, 1.2087911367416382, -1.6217173337936401, -1.1881171464920044, -0.8906521797180176, 0.3103293180465698, -1.3651158809661865, 0.27809712290763855, -1.1574434041976929, 0.0903891995549202, -0.8362240791320801, -0.6707970499992371, -0.680351734161377, 0.13366863131523132, 0.3421531319618225, 0.5406866073608398, 0.4409741163253784, -1.3635894060134888, -0.7348936796188354, -0.11513901501893997, -0.19743429124355316, 0.2782493233680725, 1.1251722574234009, -2.231823444366455, -0.7281063199043274, 0.664594829082489, -0.6673035025596619, -0.9986313581466675, -1.4349454641342163, 0.5249966979026794, -1.2348074913024902, -0.7569138407707214, 1.041102647781372, 0.4810330271720886, -0.3783189654350281, 0.2627321481704712, 0.06261507421731949, -1.6444131135940552, -0.5108876824378967, -0.7814728617668152, 0.5591607093811035, 0.4960489273071289, 1.2713196277618408, -0.5259845852851868, 0.455143541097641, -1.059659719467163, 0.4427746832370758, -0.14240321516990662, -0.03220798447728157, -0.6404926180839539, 0.845943808555603, 1.923887848854065, -0.9917552471160889, 0.09538207948207855, 0.8823482990264893, 1.4972983598709106, 0.08544677495956421, -0.34136006236076355, -0.6207048296928406, -1.6951377391815186, 1.3027151823043823, 1.342445731163025, -0.774345338344574, 1.4676463603973389, 1.772283911705017, -0.11913515627384186, -0.039164818823337555, -1.0290379524230957, 0.8276577591896057, -0.42084234952926636, -1.086567759513855, 0.09826440364122391, -1.1497886180877686, -1.14794921875, 0.8824722766876221, -0.9029254913330078, 0.7556836009025574, 1.8997995853424072, 2.6164257526397705, 1.280015230178833, -0.1916392743587494, 1.2436091899871826, 0.6049005389213562, 0.1929267942905426, 0.5821459293365479, 0.015204627066850662, -0.5768811702728271, 0.30726829171180725, -0.5487560629844666, 0.5754543542861938, 0.9855265617370605, -0.4980281889438629, -0.37279921770095825, -1.2381819486618042, 0.6366748213768005, -0.1276286393404007, -0.09626609086990356, -0.8736375570297241, -0.5652261972427368, 0.05232873931527138, 0.826847493648529, -1.7909111976623535, 1.1306419372558594, -1.6745024919509888, 0.8635661005973816, -0.24771948158740997, 0.9638045430183411, -0.6487985253334045, 1.042785882949829, -0.06781677156686783, 0.47364893555641174, -1.4358110427856445, 0.056614745408296585, -0.2840611934661865, -0.924410879611969, -0.4345676302909851, -1.0477104187011719, 1.371521234512329, -0.23459990322589874, 1.666934847831726, -0.16443578898906708, -0.6782330274581909, 3.1151111125946045, -1.2462358474731445, -1.463436245918274, 0.13408128917217255, -0.910757303237915, -1.4158930778503418, 0.0813436210155487, -1.4553501605987549, 1.2832194566726685, -0.408578485250473, -0.26683586835861206, 0.3399660587310791, 2.240044116973877, -1.8291423320770264, -1.3302901983261108, -2.2292656898498535, 0.42882633209228516, 0.7045959830284119, -0.7424020767211914, -1.7171282768249512, 0.7688304781913757, -0.3835938572883606, -0.37446877360343933, 1.857721209526062, -0.13247627019882202, -0.3945388197898865, -1.9595342874526978, -0.14396017789840698, 0.2593212425708771, 0.17958970367908478, 1.5733269453048706, -1.5414557456970215, -0.6481037139892578, -0.642858624458313, -0.3494907319545746, 0.24102255702018738, -1.0217210054397583, -3.0954461097717285, 0.385800838470459, -2.0627949237823486, 0.09571485221385956, -0.2515687048435211, 0.577701210975647, -0.17414243519306183, 1.2935534715652466, -0.13527297973632812, -1.3392187356948853, 0.5687873959541321, -1.4281456470489502, 0.058633144944906235, 0.26038309931755066, 0.13223153352737427, 0.7743533253669739, 0.4167114198207855, 0.11852633953094482, 1.038262128829956, 0.973349392414093, 0.6350642442703247, 0.42267560958862305, -0.49006685614585876, 0.013667423278093338, -0.14657637476921082, -0.7083856463432312, 0.9131286144256592, -0.8983928561210632, 1.6751859188079834, 1.8217436075210571, -1.4342737197875977, 0.7974019050598145, -2.2911641597747803, -0.4392373859882355, -0.7215763330459595, 1.2601031064987183, -0.5030048489570618, 0.03405491262674332, -1.6457804441452026, -0.5050789713859558, -1.2147597074508667, -0.7405720949172974, -0.6828900575637817, -1.1177221536636353, 0.03245516121387482, -0.27246105670928955, 1.1574528217315674, -0.9434059262275696, -0.9667838215827942, -1.1446492671966553, -1.299891710281372, -0.4010789096355438, -0.9611368179321289, -0.3872699737548828, 0.6882928609848022, -0.6092485189437866, -2.1799542903900146, -1.308609962463379, 0.0565054751932621, -0.02606172300875187, -1.1196868419647217, -0.1479772925376892, 0.8731547594070435, -0.4560476243495941, -1.6261335611343384, 0.19668589532375336, 0.8254292011260986, -1.1258536577224731, 0.8554036021232605, 0.28093644976615906, -0.8864713311195374, 1.1180636882781982, -0.8622223138809204, -0.856360137462616, 1.2605862617492676, 0.5098859667778015, -0.7887610793113708, 0.12087322771549225, 0.5385788679122925, -0.2451246678829193, 1.6785675287246704, -0.9920744299888611, 0.5224341750144958, 0.18244104087352753, 0.18560318648815155, 1.1115036010742188, 0.28602316975593567, -0.07554858922958374, -0.6842560768127441, 0.6076275706291199, 0.9121058583259583, 0.7330482602119446, -0.27874118089675903, -1.6375514268875122, -0.16453973948955536, 0.002288900315761566, 1.1278809309005737, -1.1547021865844727, -0.092012420296669, -0.22215493023395538, -1.102552890777588, 0.24116477370262146, 0.625438392162323, -0.6315325498580933, 0.4342166781425476, -2.766432762145996, -0.6150386929512024, 0.6009234189987183, 0.2932656407356262, -0.013835230842232704, -1.07229745388031, -0.4935806691646576, 0.2966586947441101, -0.18705154955387115, -0.7520780563354492, 0.997103750705719, 0.4639506936073303, -1.1964603662490845, -0.8668254613876343, 1.581611156463623, 0.18912358582019806, 2.1516318321228027, -0.3548396825790405, -0.3329669237136841, 1.7078624963760376, -0.6337236762046814, -0.6858772039413452, 0.32492950558662415, -0.44612088799476624, -0.21716804802417755, -1.259391188621521, 0.03451095148921013, 0.39004141092300415, -0.5186254382133484, -1.0648106336593628, -1.7445646524429321, 0.7980878949165344, 0.0194646418094635, -1.187610387802124, -1.9606223106384277, -0.27353259921073914, -3.475486993789673, -0.15498356521129608, -1.637666940689087, 0.31997382640838623, -0.14795826375484467, -0.15869106352329254, 0.45860496163368225, 0.735937237739563, -0.7898064255714417, 1.9614852666854858, -1.4626786708831787, 0.3447284400463104, 0.43543630838394165, 1.2569118738174438, -0.523409366607666, 0.5703141093254089, -0.6195665597915649, -0.4206237196922302, 0.9489867091178894, -0.7095323801040649, -0.5118455290794373, 0.2371169775724411, 1.3718687295913696, 0.4254615008831024, 0.6229504942893982, -0.012661273591220379, -0.6458474397659302, 0.8167474269866943, 0.4442211091518402, -1.0420544147491455, 1.2463698387145996, 0.37856900691986084, -0.4105291962623596, -1.143828272819519, -1.3479984998703003, 1.049167275428772, -2.2973170280456543, -0.23602673411369324, 0.24866971373558044, 0.31276318430900574, -1.877124547958374, 1.717169165611267, 0.8543079495429993, 0.30835971236228943, 0.8795402646064758, 0.41591933369636536, -0.6357102990150452, -1.1106294393539429, -0.9809679388999939, -1.7534699440002441, 0.7383150458335876, 0.3045945167541504, 0.19160959124565125, -0.0335683599114418, -0.5280478596687317, 1.5795176029205322, -0.6972882151603699, -1.2604589462280273, -0.2609351873397827, -1.0800666809082031, 0.358019083738327, 0.8124946355819702, 0.9272142052650452, 1.1389905214309692, 1.6381046772003174, -0.4367968738079071, -2.0162606239318848, -0.5562072396278381, -0.3231048583984375, 1.9159042835235596, 1.4395661354064941, 1.0959135293960571, 1.7788889408111572, 0.608950674533844, 0.19356203079223633, -0.21781010925769806, -0.29726487398147583, 0.7228749394416809, -0.004336505196988583, -0.1010926142334938, -0.3407767415046692, 0.4501103460788727, -2.5642073154449463, -1.2672474384307861, 1.0775890350341797, -1.0283277034759521, -0.1400473266839981, -0.3315392732620239, 1.5952047109603882, 3.3060555458068848, -0.43968403339385986, 0.7004249095916748, -0.902583122253418, 1.1646671295166016, 2.1543805599212646, 1.1264095306396484, 0.414285808801651, 0.8235001564025879, -1.1922509670257568, 0.2076384574174881, -0.15803568065166473, 0.7637305855751038, 1.2623897790908813, 1.8170286417007446, -0.02164623513817787, 0.6720293164253235, -1.7192586660385132, 0.1318158656358719, -0.0303599052131176, 0.16208025813102722, 0.6458988189697266, 0.8221228122711182, -0.5854001045227051, -1.8594857454299927, -0.048621755093336105, -1.8829948902130127, 1.6968128681182861, 1.2256815433502197, -0.08929183334112167, -0.8856274485588074, -0.8112066388130188, 0.18319006264209747, 0.8537734746932983, -0.1477600634098053, -1.922972321510315, 0.19006310403347015, -1.2988862991333008, 0.5646095871925354, -1.3220241069793701, 0.3984788954257965, 2.1759321689605713, -0.4590500295162201, 0.4643016457557678, -0.4692855775356293, -2.5045716762542725, -0.40738245844841003, -1.2355507612228394, 2.283963680267334, 0.4106130301952362, -0.023215586319565773, -1.3185983896255493, 1.885939121246338, 0.5544570684432983, -0.32412269711494446, 0.8244464993476868, -0.3058713972568512, 1.6699422597885132, -0.13903649151325226, 0.5578311681747437, -0.17065398395061493, 0.7201480269432068, -1.0555007457733154, 1.750445008277893, -0.8426170945167542, 0.23392818868160248, -0.8894069194793701, 1.235538125038147, 0.6460190415382385, -0.06142224743962288, 0.19595558941364288, -1.5731780529022217, 1.1433725357055664, 0.7177959084510803, -0.40446415543556213, -0.4005202054977417, -0.49829334020614624, 1.2503581047058105, 1.6317418813705444, 0.3210121691226959, -0.21761803328990936, 0.38231614232063293, -0.32381999492645264, 0.2614084780216217, -0.47569701075553894, 0.5926288366317749, -0.5429942607879639, 0.040918320417404175, 1.1959939002990723, -0.4271131753921509, -1.3349714279174805, -1.5377939939498901, -2.070646047592163, 0.17990532517433167, -0.8876042366027832, -0.1007649376988411, -0.07103575766086578, 0.24807696044445038, -0.558751106262207, 0.9890211224555969, 0.5125881433486938, 2.0375945568084717, 0.005604258272796869, -0.016833826899528503, -0.7654960751533508, 0.26832881569862366, 0.30040085315704346, 0.30405303835868835, -0.2976685166358948, 1.466438889503479, -0.12372736632823944, 0.5351833701133728, -1.8739466667175293, 0.6155675053596497, 1.0861098766326904, 2.3412399291992188, 0.5146074891090393, -0.4954308867454529, -1.5349137783050537, -0.8039933443069458, 1.5597140789031982, 1.083272099494934, -0.7018178701400757, -0.14586064219474792, -0.39156287908554077, 0.05619753897190094, 0.5363795161247253, -0.023369217291474342, -1.3285181522369385, 0.7345618009567261, 0.0036321328952908516, 1.4301726818084717, -0.9088453054428101, -0.43331170082092285, 0.32758817076683044, 1.3245630264282227, -0.5215954780578613, -0.8727990388870239, -0.11266247183084488, 0.44460010528564453, -0.35496482253074646, -0.6475862860679626, -0.2869996428489685, 0.7399694323539734, -1.0255917310714722, -0.2768491506576538, -0.6417595148086548, 1.18324875831604, -0.6635624170303345, 1.5502829551696777, -0.11800414323806763, -1.9323142766952515, 0.43725407123565674, 1.7447987794876099, 0.3183135390281677, 0.8748118281364441, 1.5565190315246582, 0.8153872489929199, 1.2704229354858398, -0.391094446182251, -0.08417756855487823, -1.1436712741851807, -0.20128652453422546, 1.332793116569519, 1.4634394645690918, -3.0750858783721924, -0.669907808303833, -0.4972316324710846, -1.6260452270507812, 0.76176917552948, -0.4068787395954132, -0.6100066900253296, -0.43633726239204407, -1.442294716835022, -2.3444197177886963, -0.8193975687026978, -0.29136890172958374, 2.5157413482666016, -0.2917187809944153, -0.5263627767562866, -0.3539203703403473, -1.214530110359192, 0.21319450438022614, 0.7151418328285217, -0.1228925883769989, -0.03244461491703987, 0.5562751889228821, -1.6385629177093506, 0.3422009348869324, 0.6685802936553955, 0.13738931715488434, -0.193816676735878, -1.0869078636169434, -1.2554655075073242, -0.47571447491645813, 0.8708086013793945, -0.47026553750038147, 0.4827094078063965, 0.7960856556892395, -0.8870594501495361, 0.013448819518089294, 0.25666335225105286, 0.3283728361129761, -1.1266061067581177, 2.7300686836242676, -1.3484519720077515, 1.244386911392212, -0.26468655467033386, -0.5755727291107178, 1.5606629848480225, -0.8017662167549133, 0.64922034740448, 1.109920859336853, -0.6300761103630066, -1.168652892112732, 0.21987830102443695, 1.1831873655319214, 0.8215339779853821, 0.24309617280960083, 0.874489963054657, 0.06812924891710281, 0.1787176877260208, -0.36493346095085144, -0.10970961302518845, 1.3616567850112915, 1.4637110233306885, -1.888316035270691, -2.3142566680908203, 0.03973037376999855, -0.1608782708644867, 0.8521036505699158, -1.2186192274093628, -0.4291302561759949, 0.22288918495178223, 1.869413137435913, 1.783812403678894, 0.878828227519989, 0.4492897093296051, -0.48285964131355286, -0.12978892028331757, -0.03156343474984169, -0.0756450816988945, -0.09511315822601318, 0.05678393691778183, 1.7064894437789917, -0.22576989233493805, 1.3037097454071045, -0.48080986738204956, 1.0711332559585571, 1.0297659635543823, 0.09936606138944626, -0.628007709980011, -0.3381744623184204, 3.0084946155548096, 0.899855375289917, -0.7230456471443176, 0.7865980267524719, -0.5462441444396973, -2.0771865844726562, -0.5728909373283386, 0.13556650280952454, 0.41306954622268677, 0.7273675799369812, 0.6495813131332397, -1.2198256254196167, 0.012443074025213718, 0.3037447929382324, -1.1830495595932007, -1.7276864051818848, 0.5388216376304626, -1.8594284057617188, -0.4817379117012024, -1.2144402265548706, -0.0416649766266346, -0.7761183977127075, 0.7662410736083984, 0.30362024903297424, 0.6517108082771301, 0.38305068016052246, -0.27771198749542236, -0.5411037802696228, 0.034234073013067245, -0.9038624167442322, 1.2512826919555664, -0.9093444347381592, 1.5261225700378418, -0.7485717535018921, -1.088456630706787, -0.761889636516571, 1.0237879753112793, 0.02257479541003704, 0.19506333768367767, -1.4109337329864502, -1.4592539072036743, 0.16126030683517456, -3.1862823963165283, -1.2117044925689697, -0.4396483302116394, 0.48352354764938354, 1.5575776100158691, 0.42693719267845154, -0.2967117130756378, 0.700544536113739, -0.1083204373717308, 0.9879376888275146, -0.979247510433197, -0.23627716302871704, -1.153399109840393, -0.2586910128593445, -0.8205923438072205, -0.028299331665039062, -0.3634383976459503, 0.961683988571167, 0.3696364760398865, 0.7866663932800293, 0.8837583065032959, 1.0632529258728027, 0.8177168369293213, 0.5210281014442444, -0.3190523087978363, 0.3538264036178589, 0.17093850672245026, 0.08942894637584686, -0.06321845203638077, 0.1334979087114334, -0.014532607048749924, 0.526296079158783, 0.5486840605735779, -1.8090004920959473, 0.1471753716468811, 0.3012141287326813, -2.207479953765869, -2.7062642574310303, 0.1689245104789734, -1.2530449628829956, 0.33727771043777466, -0.992341160774231, -0.15680047869682312, 1.224618911743164, -0.7211647033691406, -0.20777195692062378, 0.9619778394699097, 0.2988796532154083, -1.6125540733337402, -0.6413714289665222, 0.02899160049855709, -2.211886167526245, -0.7881370186805725, 1.8350263833999634, -1.3305563926696777, -0.6607018113136292, -0.6981414556503296, -1.4105223417282104, 0.006077664438635111, -0.6401416659355164, -0.6369116306304932, -0.40536820888519287, -0.006720155477523804, 0.1365160495042801, -0.734978199005127, 0.8345237970352173, 0.6647194027900696, 1.3308643102645874, -2.845463991165161, -1.8362571001052856, 0.8466317653656006, 0.4954862594604492, -1.211790919303894, 0.9228507876396179, 0.8953425288200378, -0.052687548100948334, -0.4426456689834595, 0.012101421132683754, 0.37675705552101135, 1.3766131401062012, -0.2056756168603897, 0.3948691189289093, 0.6043050289154053, -0.29856905341148376, 0.7577611804008484, -3.0144894123077393, -0.30925023555755615, -0.5777881145477295, -0.30410709977149963, -0.8155368566513062, 0.6503031253814697, -1.4109114408493042, 2.5544052124023438, -0.4550548195838928, 0.1957290768623352, 0.08465475589036942, 1.34232759475708, 0.12064716219902039, -0.6610381603240967, -0.3566751778125763, 0.7574641704559326, -2.499570608139038, -0.9626180529594421, -1.0194638967514038, -1.7845895290374756, -1.3429518938064575, -0.04447144269943237, -1.467535138130188, 1.1440824270248413, -0.45877495408058167, 0.8990684151649475, -0.011574606411159039, 0.6811073422431946, -1.089558720588684, -0.720508337020874, 0.3381781280040741, -0.07856230437755585, 0.9400724172592163, 0.9742640256881714, 0.49658870697021484, 0.514398455619812, -0.3806155323982239, 0.7115808725357056, -0.7830613255500793, -0.7732297778129578, 1.5839866399765015, -0.35806578397750854, 0.737783670425415, 1.1206517219543457, 1.1414250135421753, -0.42351847887039185, 0.38268446922302246, -0.07063405215740204, -0.8249382972717285, -0.8497559428215027, 0.7969977855682373, 1.0221850872039795, 0.5508121848106384, -2.1214988231658936, -0.38500818610191345, -0.30122110247612, -1.131005048751831, 0.16680319607257843, -0.6351451277732849, 1.0175707340240479, 0.48155349493026733, 0.38449135422706604, 0.039294853806495667, 1.0054677724838257, -1.4428489208221436, -0.583768904209137, 0.8687391877174377, -0.6611480712890625, -0.01563357375562191, 1.3659306764602661, 0.12177756428718567, -1.1278570890426636, -1.9492874145507812, 0.5754680633544922, 0.012339483015239239, 0.42224985361099243, -0.3767281174659729, 2.817253828048706, 0.05815226957201958, -0.5754693746566772, -1.1674156188964844, 0.5158780813217163, 0.664214551448822, -0.11302950978279114, 0.7103937268257141, -0.9503495693206787, -0.6895068883895874, -0.5252442955970764, 0.5507341027259827, 0.8649409413337708, -1.8311623334884644, 1.2007468938827515, 0.21533165872097015, 1.6805075407028198, 0.3059755265712738, 0.22394931316375732, -0.5839329361915588, 0.6550732254981995, 0.498939573764801, -1.027007818222046, 1.3424748182296753, -0.7741836309432983, -0.7180075645446777, 1.4227076768875122, -1.0467528104782104, 0.01748284325003624, 1.4569025039672852, 0.2140950709581375, 0.4145384728908539, 0.6246080994606018, -0.7031956315040588, 0.3111543357372284, 0.32057419419288635, -1.4368937015533447, 1.7283809185028076, 1.063761591911316, 1.2684729099273682, -1.5673272609710693, -0.8590090274810791, -1.537878394126892, -0.011259432882070541, 1.9681063890457153, -0.5489049553871155, 0.6073157787322998, -1.3471463918685913, 1.5695087909698486, 0.6133284568786621, -1.2864904403686523, -1.9996695518493652, -0.16599707305431366, 0.2588772475719452, -0.18835091590881348, 0.6059433221817017, -0.1181013286113739, 1.2215766906738281, -0.8472756743431091, -0.4858059883117676, -1.1304843425750732, -0.06752043962478638, 2.104321241378784, -1.1682603359222412, 0.5403775572776794, -0.3863283693790436, -0.045915648341178894, 0.9705334901809692, -1.1937248706817627], [-0.2648202180862427, -0.7289759516716003, 0.16084197163581848, -0.9941478371620178, -0.6896008253097534, -1.091330885887146, 1.1439989805221558, 0.08400976657867432, -0.9107135534286499, -1.00278639793396, -0.32624351978302, -0.46730801463127136, 0.1879291534423828, -0.4836929142475128, -0.46843186020851135, 0.4931650757789612, -1.6050152778625488, -1.2164983749389648, 0.7521259188652039, -0.2542518079280853, -0.3961816728115082, -1.4410579204559326, -0.8702830672264099, 1.5400521755218506, 0.7981897592544556, 1.3854422569274902, 1.0213499069213867, 0.8293190598487854, 0.58493572473526, 0.37060675024986267, 0.08127526193857193, 0.6067838072776794, 0.861690104007721, -1.5190805196762085, 0.14242352545261383, -0.8120089769363403, 0.6890798807144165, 0.27460777759552, 0.1781112104654312, 1.2162837982177734, 0.6275254487991333, -0.39093002676963806, -0.3130832016468048, 0.24499846994876862, 0.3267495036125183, 0.2322663962841034, -2.0417559146881104, -0.28869733214378357, 1.8779776096343994, -0.34612369537353516, 1.4798460006713867, 1.332533836364746, -0.6304976344108582, 1.1346021890640259, 0.4305709898471832, -0.2256624698638916, 0.15042045712471008, 1.4566514492034912, 1.107154369354248, -0.9120522141456604, 1.442125916481018, -0.33285683393478394, -1.3670881986618042, -0.3834628462791443, 1.3792959451675415, -1.4901156425476074, -0.2596319317817688, -2.098816156387329, -0.15215416252613068, 0.6910172700881958, -0.9270181655883789, 0.510428786277771, -0.6432999968528748, -0.9375525712966919, 0.30008572340011597, 1.0694128274917603, -2.3992011547088623, -0.053706586360931396, -0.8907304406166077, -0.28292813897132874, -0.17632880806922913, 1.136319637298584, 1.307952880859375, 0.28679367899894714, 1.0462005138397217, 1.5435864925384521, -0.6994485259056091, 1.2352219820022583, -1.0756455659866333, 0.593260645866394, 0.7920107841491699, 1.2731443643569946, 0.4345628321170807, 0.7771037817001343, 0.9924558401107788, 0.8003872632980347, 0.07348163425922394, 2.01326847076416, -0.8595910668373108, 0.20087552070617676, -1.2382583618164062, 0.5682320594787598, 0.28113797307014465, 0.7750054597854614, 0.2406761646270752, 2.2236406803131104, 1.006003975868225, 0.0853288322687149, 0.418384850025177, -0.3444598317146301, -0.9488398432731628, -0.6701740026473999, 0.09681984037160873, -1.7704893350601196, 0.3534933924674988, 0.7545463442802429, 0.29469290375709534, -0.4697866141796112, -0.4556863307952881, -1.2977616786956787, 0.7304962873458862, 0.02269313670694828, 1.8919615745544434, 0.3841528594493866, 0.5599362254142761, 0.4628608822822571, 0.6714512705802917, -0.5551370978355408, -0.1929074376821518, -1.5911941528320312, 0.3852784037590027, 0.2167092263698578, -2.2295899391174316, -0.8248059749603271, 1.3448622226715088, 0.4644409120082855, -1.0249851942062378, -0.8424372673034668, -0.05931418389081955, 0.9125155806541443, 1.1445826292037964, 0.933443546295166, 1.3034336566925049, 1.400846242904663, -1.0488122701644897, 0.5341164469718933, -0.9241791367530823, 0.259316086769104, 1.2316927909851074, -0.7767366170883179, 2.0619912147521973, -0.8422870635986328, 0.8184046745300293, 0.597657322883606, -1.673617959022522, 0.45291006565093994, 1.7857484817504883, 0.4996856451034546, -0.31590062379837036, -0.39835092425346375, -1.3642518520355225, 0.5617658495903015, -0.6321311593055725, -0.09767485409975052, -0.3782002031803131, 0.31309157609939575, 0.2750110924243927, 1.8084443807601929, 0.3238053619861603, 0.07867913693189621, -0.4223558008670807, -1.8373117446899414, 0.9193307757377625, 1.469666838645935, -0.07588192820549011, -0.2240850180387497, -0.5363169312477112, 0.6518774628639221, -0.6396099328994751, 0.23047170042991638, -0.09049203991889954, 2.3181393146514893, 1.7777255773544312, -0.025155698880553246, -0.3793703615665436, -0.9802833199501038, -1.0046714544296265, -0.0671016126871109, -0.2992711663246155, 0.2781374454498291, 0.6055325269699097, 1.3615845441818237, -0.51609206199646, -2.1249797344207764, 0.6329838633537292, -0.45567989349365234, 0.32832372188568115, -0.09614495933055878, 1.8917014598846436, -0.9114862084388733, -1.4468450546264648, -2.569051742553711, -0.09158484637737274, 0.7341417670249939, 1.3528852462768555, -1.274316668510437, 1.4618498086929321, 0.9015396237373352, -1.2742950916290283, 0.8659304976463318, 0.5006091594696045, 1.8394581079483032, -0.17345212399959564, -0.3050817847251892, 0.8795490264892578, 1.8511525392532349, -1.0950536727905273, -0.43573904037475586, 0.0803012102842331, -0.6329925656318665, -0.25059112906455994, -0.911562979221344, -0.5767558813095093, -0.2905633747577667, -0.53321772813797, -0.2630888819694519, 0.5482211112976074, -0.15065643191337585, -1.0359625816345215, 1.2724369764328003, 0.3228245973587036, -0.4858849346637726, -1.116020679473877, 0.019329849630594254, 0.492539644241333, -0.6457028985023499, -0.6417478919029236, 0.3356010317802429, 0.079556405544281, 0.29989221692085266, -0.5597559809684753, -1.0670124292373657, 0.1714829057455063, 0.5740106105804443, -0.5561230778694153, 0.2544158399105072, -1.5442003011703491, 0.44007691740989685, -1.9827231168746948, -1.116767168045044, 0.28309524059295654, -0.04649203643202782, -1.1256964206695557, 0.22925284504890442, -0.2963801324367523, 0.5220749974250793, 1.8370600938796997, 0.6658284068107605, 1.034928560256958, 0.21179912984371185, -0.20326422154903412, -0.2786661684513092, -1.2996944189071655, 0.3296293616294861, 0.8220393657684326, -0.839023232460022, -0.27512672543525696, -0.9032257795333862, -0.608971357345581, -0.1406431794166565, 0.45506247878074646, -0.26677531003952026, 0.08925776928663254, -1.9340972900390625, -0.582956075668335, -1.0016175508499146, 0.164981871843338, -2.4213874340057373, 0.952965497970581, -0.3674531579017639, 0.16258162260055542, 0.21031974256038666, -0.6921516060829163, 0.11339569836854935, -1.7201389074325562, 0.5245093107223511, -1.873987078666687, -2.0887205600738525, 0.38353729248046875, 0.4156095087528229, 0.5275994539260864, -1.5126608610153198, 0.5670879483222961, 0.8431290984153748, 0.858031690120697, -0.8580377101898193, 1.328486442565918, 1.0494260787963867, -2.0440328121185303, 1.3171831369400024, 2.5802865028381348, 0.6787788271903992, 0.9554670453071594, 0.1465454250574112, -0.18773090839385986, -0.17667753994464874, -2.8008198738098145, 0.4973694682121277, 0.6939921975135803, -1.6184226274490356, 0.5494964122772217, -0.245162233710289, 1.5598418712615967, -1.6509565114974976, 0.1123235896229744, 1.7800143957138062, 0.699976921081543, -0.10421343892812729, 1.0121670961380005, 1.2917026281356812, -0.6817052960395813, -0.8743324279785156, -0.22123876214027405, 0.21573618054389954, -0.27095678448677063, -0.14390592277050018, 0.7326172590255737, -0.13091403245925903, 1.1443572044372559, -0.33840665221214294, 1.4491232633590698, 1.6661330461502075, 1.1846529245376587, 0.2971384823322296, 1.1679965257644653, 0.8072139620780945, -0.9289964437484741, 0.9060195684432983, -0.9599238634109497, -1.315878987312317, -0.9151227474212646, 0.15258561074733734, -1.754549503326416, 1.7988321781158447, -0.5745615363121033, -0.22098591923713684, 0.35881081223487854, -1.6673848628997803, 0.3586259186267853, -0.32122817635536194, -1.0722746849060059, -1.5433790683746338, -0.9367305040359497, -0.5225905776023865, -0.6087262034416199, -0.2664843499660492, 0.46725502610206604, 0.2823346257209778, 0.6120161414146423, 0.5025722980499268, -0.7842887043952942, -1.4493682384490967, 0.7109681963920593, -0.3974246084690094, -1.3369908332824707, -0.6402241587638855, -0.2556675970554352, -0.22350263595581055, -0.4304860830307007, -0.46126753091812134, 0.1087968721985817, -0.024954726919531822, 0.04949729144573212, 1.5481023788452148, -0.5810784101486206, -0.39069798588752747, 1.371559500694275, 3.3306782245635986, -0.045210547745227814, -0.3957071006298065, 1.161215901374817, 0.5396093130111694, 0.6610466837882996, 1.1616911888122559, 0.0980149433016777, -0.32935285568237305, 0.5951669216156006, -0.6085835695266724, 1.7132755517959595, -0.851658821105957, 0.49153637886047363, 0.07708948850631714, 0.5304884314537048, -1.1827826499938965, -0.6631940007209778, -0.21363341808319092, 0.9569748044013977, 0.793494701385498, 1.4922382831573486, 1.123542308807373, -0.650270938873291, 0.6833577156066895, 1.8263534307479858, -0.23299771547317505, -0.8729906678199768, -1.1199597120285034, -0.8210583925247192, 0.8145270943641663, -1.3294016122817993, 0.8531323671340942, -1.9004393815994263, -1.0641887187957764, 2.3243210315704346, -1.7489206790924072, 0.491409033536911, 0.06766657531261444, 0.3301868140697479, -1.6535168886184692, 0.2714471220970154, 0.06341532617807388, -0.10381657630205154, -0.29174596071243286, 0.4620739221572876, -0.05819215625524521, -0.6307761669158936, 0.06547997891902924, -1.056096076965332, 1.4901384115219116, 0.8571810126304626, -0.7137561440467834, 0.6896023750305176, -0.3266637623310089, 0.3008449673652649, 0.29677918553352356, 0.0068967100232839584, 1.1554864645004272, -0.4984482228755951, -0.1696496158838272, -0.4112555682659149, -0.6625373959541321, -0.18237122893333435, -0.8328009247779846, -1.0267388820648193, -0.5987427830696106, -1.4514726400375366, -0.9862414002418518, -1.4730430841445923, 0.20388667285442352, -0.3542024791240692, -0.283797025680542, -0.24904762208461761, -0.8499279618263245, -0.306271493434906, -0.05952199175953865, 0.7732285857200623, -0.33977386355400085, 0.4599708318710327, 0.317609578371048, 0.0026329075917601585, -0.6120729446411133, 1.5029206275939941, 0.39257141947746277, 0.19896014034748077, -1.7954529523849487, 0.7173594236373901, 0.08038754761219025, 0.5700619220733643, -0.4986128807067871, -0.6797522306442261, -0.24975523352622986, -0.8029184937477112, 0.7878198027610779, 0.48596495389938354, 0.26224684715270996, -0.44865232706069946, 0.2945184111595154, 0.2934463322162628, -1.819108009338379, 1.7268916368484497, -0.09755115956068039, -1.4648535251617432, -0.15044517815113068, 0.8400318026542664, -0.2880162000656128, -1.9787455797195435, -0.018784934654831886, 0.22236807644367218, 0.6651675701141357, 0.514153003692627, 0.0523926205933094, 0.9181002378463745, 1.2875219583511353, 0.5780630707740784, -0.8784835338592529, 1.368435025215149, -1.5291374921798706, -0.24980176985263824, -0.6980628371238708, 0.684183657169342, -0.7435441017150879, -0.7518435716629028, 0.8408094644546509, 0.4636560380458832, 0.11594845354557037, -0.584187388420105, 0.057351306080818176, -1.1993862390518188, 0.2282901108264923, 1.7697573900222778, 0.7012898325920105, -0.24786141514778137, 0.5998481512069702, -0.3086656630039215, 0.11822855472564697, -0.6404592394828796, -1.1585767269134521, -0.14701421558856964, 0.7282905578613281, -1.8530447483062744, -1.032939076423645, 1.226579189300537, 0.8693302273750305, -1.1625583171844482, -0.365610808134079, 1.9435800313949585, 0.06010958552360535, -1.1735527515411377, 0.8369683623313904, 0.8472614288330078, -1.265783429145813, 1.4883437156677246, 0.18874789774417877, -1.1008416414260864, 0.712263286113739, 0.5489934086799622, 0.4100072383880615, 0.30002298951148987, -0.9513735771179199, 0.6259626150131226, -0.6105624437332153, 0.15878795087337494, -0.6982530951499939, 0.06595636904239655, 1.5406603813171387, 0.33787062764167786, 2.1240172386169434, -0.8800640106201172, -0.19836053252220154, 0.32941344380378723, 1.7654874324798584, 0.4720754325389862, -0.41079476475715637, 0.8243955373764038, 0.09546324610710144, 0.7341939210891724, -0.38530391454696655, -0.5550370812416077, -0.7446846961975098, 0.0038525147829204798, 0.8618335723876953, -0.05625567585229874, 0.8462555408477783, -0.966616690158844, -0.47724777460098267, -0.8579672574996948, -0.8525913953781128, -0.554343044757843, 0.04590161144733429, -1.8632478713989258, -0.806125283241272, 0.8030018210411072, -0.20923887193202972, -0.8312485814094543, -0.4895329177379608, -0.14312845468521118, -0.7602035403251648, -0.6040757894515991, 1.5416136980056763, -2.5216426849365234, -1.3146249055862427, 0.9115146994590759, 0.32814839482307434, -0.2741626501083374, -1.124984622001648, 1.0610419511795044, -1.647580623626709, 0.21529515087604523, 0.09777843207120895, 0.1410391628742218, -0.962793231010437, -0.7701170444488525, 0.1862015575170517, 0.3214494287967682, -0.06770053505897522, -0.7114331126213074, -0.8305440545082092, 0.0412641279399395, 0.6038935780525208, 1.5419085025787354, -1.137103796005249, -0.17811468243598938, -0.03994613513350487, 1.6294137239456177, -0.09568635374307632, -0.32561594247817993, -1.3120487928390503, 0.15955336391925812, -0.6057014465332031, 0.8224450945854187, 0.525708019733429, -1.7498904466629028, -0.7391881942749023, -0.14324139058589935, 0.3299902081489563, -0.15786783397197723, -0.30699729919433594, -1.2189195156097412, 0.6437358856201172, -0.2013496607542038, -0.349056214094162, -0.15412203967571259, 0.1789880245923996, -0.13616755604743958, 1.0878413915634155, 0.250186026096344, 0.2758682072162628, -0.7131443023681641, -0.7686389088630676, -0.8861930966377258, 0.14502006769180298, 0.2994144558906555, 0.5348567366600037, -0.07672625780105591, -0.2525099515914917, -0.022617999464273453, 2.3087596893310547, -1.2317798137664795, 0.02029537595808506, 0.3991571068763733, 0.5594542622566223, -1.9823479652404785, 0.3477700650691986, -0.40483173727989197, 0.0008889530436135828, 1.0171924829483032, -1.276702642440796, 0.45392918586730957, -0.4926671087741852, 0.3059852719306946, 1.1346077919006348, 1.1852858066558838, -1.2507771253585815, 0.6293273568153381, 1.6159005165100098, -0.022261153906583786, -0.7207640409469604, 0.3705858886241913, -0.5073330998420715, -0.3534737825393677, -1.3834245204925537, -0.6702650189399719, -0.2919163405895233, -1.589025616645813, 1.4853942394256592, 0.6059466600418091, 0.7349418997764587, 1.0421735048294067, -0.45185890793800354, -1.9067678451538086, 0.6503615975379944, -0.7970055341720581, -0.14745864272117615, -0.7989062070846558, -1.5147720575332642, 1.74623703956604, -0.5130425691604614, -0.9919512271881104, 1.722388744354248, 1.5770673751831055, 0.05574454739689827, -1.032594919204712, 0.6960591077804565, -1.077380657196045, 3.001192331314087, 0.2811996638774872, 0.40484383702278137, -0.9638354182243347, 0.5441492795944214, -1.0179096460342407, 2.3770806789398193, 0.5351172089576721, 0.24293260276317596, -0.6898824572563171, 0.49208804965019226, 0.889640212059021, 0.4916705787181854, 1.5342226028442383, -0.7711887359619141, 0.27112436294555664, 1.0048940181732178, 0.23993636667728424, 2.1983702182769775, -1.4172800779342651, 0.6403952836990356, 0.3301040530204773, -0.30732259154319763, 1.6799098253250122, -0.028742900118231773, 1.0703721046447754, 0.7409318685531616, -0.3528164327144623, -0.9842022061347961, 0.2508576214313507, -0.2858583331108093, -1.3313833475112915, -1.4177298545837402, -1.8030678033828735, -2.186832904815674, 0.28475433588027954, -0.5599784851074219, -0.703555166721344, -0.7087631821632385, -0.5630279779434204, -1.2081907987594604, 0.18605558574199677, -0.7925024032592773, -1.235154390335083, -0.6622257232666016, -0.39732614159584045, -1.5554717779159546, 0.48755595088005066, 0.5944347977638245, 0.9134510159492493, 1.4639590978622437, -1.0621238946914673, -0.6900946497917175, -0.08706726133823395, 1.8117730617523193, 0.2639956772327423, -0.25102439522743225, -1.094777226448059, -0.2848758101463318, -0.5663483738899231, -0.8913350105285645, -0.10625311732292175, -0.3240389823913574, 0.6188969612121582, 1.3290590047836304, 0.9748672842979431, -1.0811710357666016, -0.43390530347824097, -0.7950615286827087, 0.5161484479904175, 0.45383211970329285, -1.2848588228225708, 1.0293346643447876, -0.612210214138031, 1.2268710136413574, -0.37519311904907227, -1.9353452920913696, -0.1388603150844574, 1.3288865089416504, 1.794862151145935, 0.7440037131309509, 0.6017788648605347, 0.387315958738327, 0.3749723434448242, 2.1285362243652344, -1.4245480298995972, -0.5932053923606873, 0.35365697741508484, 0.3309893310070038, -0.009755024686455727, 1.3513621091842651, -0.885291576385498, 0.16519631445407867, -0.3863048553466797, -0.3275775909423828, 0.8167847394943237, -1.3803683519363403, 0.06528714299201965, -0.1047322154045105, 1.3967740535736084, 0.025806991383433342, -0.8038480281829834, 1.1510919332504272, 0.9543235301971436, -1.0073304176330566, 0.11603489518165588, -0.479823499917984, -0.3288866877555847, -1.1018037796020508, 0.5242888331413269, 0.8177696466445923, -0.24218320846557617, 0.6046098470687866, -0.2758869230747223, 0.37611815333366394, 0.809674859046936, 1.526135802268982, 0.6343784332275391, 0.17074166238307953, 0.1844938099384308, 2.3133835792541504, 0.15318025648593903, -0.8473421335220337, -0.7268534898757935, -1.05637526512146, 0.34680265188217163, 0.6035305857658386, -0.7610237002372742, 0.2839216887950897, 0.3759249150753021, -0.521577000617981, -1.1389577388763428, 0.29911136627197266, 1.2210074663162231, 1.7721962928771973, -0.4153328537940979, 0.5900434255599976, -0.18051573634147644, -0.08767884969711304, 1.2909188270568848, -2.026442766189575, 0.028937026858329773, 0.33200010657310486, -0.8619539141654968, 0.8043705224990845, -0.2467672973871231, -0.7061648964881897, 0.49661144614219666, -0.43635550141334534, -0.31999471783638, 1.9215737581253052, -0.22233471274375916, 0.22916239500045776, 0.2530997395515442, 0.7021427750587463, 2.393834114074707, 0.40206608176231384, 0.45912614464759827, -0.02940662018954754, 0.7711620926856995, 1.932889699935913, 0.5686768889427185, 1.9484351873397827, 2.0034663677215576, 0.9053384065628052, 0.27591440081596375, 0.14362883567810059, -0.8347966074943542, -0.8188949227333069, -0.6824647784233093, -1.3343169689178467, -1.0769050121307373, -0.07090061902999878, 0.23884159326553345, -0.47589239478111267, -1.2179166078567505, -0.3513358533382416, -0.15768837928771973, -0.324740469455719, -0.34167513251304626, -0.6458408236503601, 0.15839087963104248, 0.025820227339863777, 0.4094580411911011, -0.49395811557769775, 0.41654226183891296, 0.17885680496692657, 0.8324263095855713, 0.304107129573822, 0.7169992923736572, -0.4862435460090637, -0.6785808205604553, -0.3252531588077545, 0.07175961881875992, 1.697681188583374, 1.7707552909851074, -0.14446713030338287, 0.8575560450553894, 1.1501493453979492, 0.6511363983154297, 0.34707438945770264, -0.2528628706932068, 0.32503560185432434, 1.2222330570220947, 0.19170379638671875, -1.3128082752227783, -1.6761788129806519, 0.6957769393920898, -0.26450276374816895, -0.4148849546909332, -0.03211413323879242, -2.864053726196289, 0.4972551763057709, -0.9489606618881226, -3.765871047973633, 0.14290641248226166, 1.0046733617782593, -1.34633469581604, -0.8455243706703186, 0.7756837010383606, -1.6439313888549805, 1.6944454908370972, 1.9044711589813232, 1.7143863439559937, -0.30572253465652466, 0.29701775312423706, -0.39928463101387024, 0.1959037333726883, 1.3021734952926636, 0.7191010117530823, -0.10970569401979446, 0.8756816983222961, 0.5112593770027161, 1.0454503297805786, 1.2601033449172974, -1.0417386293411255, -1.1470584869384766, -0.6028943657875061, 0.9647411704063416, -1.5291658639907837, -1.9777612686157227, 0.1692827343940735, 0.24557602405548096, -1.3867292404174805, -0.280315101146698, 0.23472349345684052, -1.2656800746917725, 0.6481907963752747, -0.9066532254219055, 1.04072105884552, 1.027919888496399, -1.5041677951812744, 1.36526358127594, -1.6355384588241577, 0.7572408318519592, 0.4398736357688904, -1.058786153793335, -1.1155370473861694, -0.32195237278938293, -0.3722447156906128, -0.5448423624038696, 0.7226563692092896, 0.5466620922088623, -0.03118826448917389, 0.4139849841594696, 0.5429191589355469, -0.814659833908081, -0.8449766635894775, 2.20406174659729, 1.1886893510818481, -0.09639065712690353, 1.9828439950942993, -1.5345381498336792, 0.2970863878726959, -1.0654476881027222, 0.7271894216537476, 1.9667943716049194, 1.2301543951034546, 1.261895775794983, -0.28556251525878906, 0.36192232370376587, 0.5962082147598267, 0.6927476525306702, 0.49988722801208496, -0.21196337044239044, -0.4097738564014435, 1.715552568435669, 0.11304975301027298, 2.0536253452301025, 0.4829864203929901, 0.1279936283826828, -1.8712315559387207, 0.008396955206990242, -0.20096886157989502, -0.23715448379516602, -1.3782262802124023, -0.6155112385749817, 2.0391619205474854, 0.40178483724594116, 0.6992167830467224, 0.44224125146865845, -0.38563913106918335, -1.425647497177124, 0.6283218860626221, -0.05842707306146622, 0.04111502692103386, -0.29565244913101196, 0.5112811923027039, 1.1916577816009521, 0.3086915910243988, 0.046373456716537476, 0.9768103957176208, 0.6693176627159119, -0.21251846849918365, -0.2243574559688568, 0.6462435126304626, -0.702578604221344, -1.0198750495910645, -0.4825056791305542, -0.7827402949333191, -0.5308261513710022, -0.6134450435638428, 0.5900951623916626, -0.15495236217975616, 0.23611411452293396, 1.0111795663833618, -0.693951427936554, 0.11079859733581543, 0.5001229643821716, -0.978678286075592, -0.745526909828186, 0.6439735889434814, 2.4618752002716064, 0.8824936747550964, 0.8516539931297302, 1.1320215463638306, 1.3484925031661987, -1.7907413244247437, -0.9621830582618713, -0.435621440410614, -1.5428547859191895], [1.097002625465393, -0.1559460610151291, 0.22982124984264374, -1.2327650785446167, 0.27091893553733826, 0.7978054881095886, -0.7637603878974915, -2.57173228263855, 0.135450080037117, -0.12154179811477661, -0.12679259479045868, -0.14947274327278137, -0.1226048395037651, 1.0320724248886108, 0.6551846265792847, -1.2252166271209717, 0.9140715003013611, -0.7245651483535767, 0.6604673266410828, -0.7056524753570557, -0.12986716628074646, 0.982574999332428, -0.3603987395763397, 0.9565768837928772, 0.495033323764801, -2.363407850265503, 0.6642788648605347, 0.7581280469894409, 0.6590083241462708, -1.1474608182907104, -1.626261591911316, 1.12160325050354, -0.31786012649536133, 0.25060832500457764, -1.0876339673995972, -1.7622793912887573, 0.6898834705352783, 0.24485965073108673, 0.9675154089927673, -1.0832287073135376, -0.6989105343818665, 0.06280665099620819, -0.018743589520454407, -0.006712142378091812, 1.4135005474090576, 0.5714626908302307, -0.5454413890838623, 0.10984379053115845, 1.4643880128860474, 0.15643183887004852, -0.9680773019790649, 0.014656840823590755, 0.16827630996704102, 1.018786072731018, -0.1421053260564804, 0.11343345791101456, 0.6614017486572266, 1.2553726434707642, 0.0736716017127037, -0.2764696478843689, 0.006483603268861771, 0.7851387858390808, -0.34232792258262634, 1.5940226316452026, 0.6654790639877319, -0.456226646900177, 0.12933599948883057, -0.47954311966896057, -0.6858460903167725, -1.3051576614379883, 0.9728246927261353, -0.3587707281112671, -1.5656557083129883, 1.0213367938995361, 0.4188286364078522, 1.4146497249603271, 1.133787751197815, 0.38314613699913025, 1.732089638710022, 1.5736045837402344, -0.4114826023578644, -1.2606725692749023, 1.2978633642196655, -1.3385016918182373, -0.16886301338672638, 1.7228171825408936, -0.41876158118247986, -0.21263504028320312, -0.31472811102867126, 0.17317622900009155, -0.09632093459367752, -0.7552101612091064, -0.014447630383074284, -0.14600712060928345, 1.0082461833953857, -1.228339672088623, 1.3680890798568726, -1.4925669431686401, 0.1317528784275055, 1.0821865797042847, 0.7242920398712158, 1.1278140544891357, 0.24886232614517212, 0.630191445350647, 0.5083510875701904, 0.16260090470314026, -0.03812340646982193, -0.5461874008178711, 0.9942439198493958, 0.7426761984825134, -0.33345943689346313, 0.29213544726371765, -0.5940424799919128, -0.1248687282204628, 0.570219874382019, -0.5072897672653198, 0.9277082681655884, 1.6573323011398315, -2.491802930831909, -0.6246769428253174, -0.8319913744926453, 3.1822075843811035, -0.30429551005363464, 0.020003343001008034, 0.7314141988754272, 0.1423957794904709, -0.4764806032180786, -0.6383073329925537, 0.6856474876403809, 1.7076259851455688, 2.140599489212036, 0.757881224155426, -0.1107323095202446, -1.8537545204162598, -0.8923031687736511, -2.4487032890319824, 0.8011677265167236, 2.0937514305114746, -1.4692071676254272, -0.7856352925300598, 0.7305408716201782, 0.6825780272483826, 0.08099794387817383, 0.1880543977022171, 1.274614691734314, -0.6968634724617004, -1.2197234630584717, 0.7166593074798584, 0.21815109252929688, -0.09692703187465668, 0.5735415816307068, -0.05846896767616272, 1.2789818048477173, -1.4436087608337402, 0.5857253670692444, 1.0278499126434326, 1.1859402656555176, 1.3610410690307617, -1.6261333227157593, 0.6436058282852173, -0.7546437382698059, -0.31552889943122864, -0.7363656759262085, 0.38938453793525696, 0.6858360171318054, 0.01839507929980755, 1.1083998680114746, 1.0334008932113647, 0.9550022482872009, -0.19603237509727478, -0.5403757095336914, -0.6522376537322998, 0.8192245960235596, 0.5801278352737427, -0.4228111803531647, 0.40480902791023254, 0.4241493344306946, 0.551787793636322, -1.197670340538025, -0.6342127323150635, 0.3020848035812378, 0.4919874370098114, -1.4144172668457031, -0.3745185434818268, -0.6027261018753052, -2.2055039405822754, -0.6156708002090454, 0.7111014127731323, -0.8813077807426453, -0.6909851431846619, -0.7745509743690491, 0.6556167602539062, -0.8635664582252502, -0.25056931376457214, 0.8865535259246826, 0.22653722763061523, -0.7041260600090027, 0.3085757791996002, 1.6082754135131836, -1.0664488077163696, 0.8490464687347412, 0.30094870924949646, 1.851070523262024, 0.2072973996400833, 1.4669438600540161, 1.5783557891845703, 1.170017957687378, -0.20439651608467102, -0.5357795357704163, -0.964375913143158, -1.3117256164550781, -0.09333249181509018, -0.006066559348255396, 0.9179169535636902, -1.1243021488189697, 0.9809526205062866, -0.3306802809238434, -0.4675588011741638, -0.31341883540153503, 0.23828406631946564, 0.30952146649360657, 0.27946987748146057, 2.0225882530212402, -1.12497878074646, 1.3001548051834106, -1.1463146209716797, 0.9637570977210999, -0.3554427921772003, -1.071366310119629, -1.1819112300872803, 1.409784197807312, -1.4748724699020386, -1.0068358182907104, -1.5478228330612183, -2.4833242893218994, -0.4139952063560486, 1.0388622283935547, 0.2560496926307678, 0.38064783811569214, -0.31233304738998413, 0.8658192753791809, -2.128873586654663, -1.6863867044448853, 0.43507370352745056, 0.07891730219125748, -1.3337606191635132, -1.8953092098236084, 1.0240782499313354, -0.5879031419754028, -0.19779866933822632, 1.5730277299880981, 0.5705330967903137, 0.9959525465965271, -0.3008444905281067, -0.5153164267539978, -0.649848461151123, -0.5256397128105164, -1.2654284238815308, -0.7397606372833252, -0.31955254077911377, 2.721498966217041, 1.455775260925293, -0.537982702255249, 0.20667831599712372, 0.3529481589794159, -1.5156376361846924, 0.5380261540412903, 0.13111178576946259, 2.463437557220459, 0.9409807920455933, 0.423861563205719, -0.04992208629846573, -0.37329381704330444, 1.046486735343933, -1.597084403038025, 1.5923088788986206, -0.7737641334533691, 0.4466208517551422, 0.6590290665626526, 0.3104020357131958, 0.23995360732078552, -0.7358741760253906, 0.179751455783844, 0.43825745582580566, -0.2947404682636261, -0.11444249004125595, 0.6043551564216614, 1.5717204809188843, -0.5441240072250366, -0.1925615668296814, 0.29427817463874817, 0.25578898191452026, -1.3223179578781128, 0.7963473796844482, -0.4678407907485962, -1.2528678178787231, 0.030564019456505775, 0.17840874195098877, 1.2165679931640625, 0.23907800018787384, 0.21273979544639587, 0.4926126003265381, 1.537549614906311, -0.3421573340892792, -1.5292128324508667, -0.5969562530517578, 2.1641640663146973, 0.6152934432029724, 0.17632931470870972, 0.12194788455963135, -1.0896574258804321, -1.7421385049819946, -2.465876579284668, -0.7480841279029846, 0.12052836269140244, -0.005300952587276697, 0.33961716294288635, -0.9868322014808655, 1.271777629852295, 0.4690661132335663, 0.20711255073547363, 0.6053625345230103, 0.27733883261680603, -1.056459903717041, -0.1549099236726761, 1.5170326232910156, 2.2247331142425537, -0.5782001614570618, 1.1198797225952148, -0.5519195795059204, -0.2663756012916565, 0.6853054761886597, -0.20464423298835754, 0.04137451946735382, 0.5543100833892822, 1.2158081531524658, 1.3791613578796387, -0.7940348982810974, 1.5775055885314941, 1.4492897987365723, -0.3198012113571167, -0.5399550199508667, -0.6157687306404114, -0.46914926171302795, 1.957709550857544, -1.0672427415847778, -1.187883734703064, -0.7760274410247803, -0.5781259536743164, -0.2840788662433624, 0.1932135671377182, -0.09666646271944046, -1.6727217435836792, -1.6355323791503906, 0.22011558711528778, 0.0863250270485878, -0.8008875846862793, -1.145459771156311, -1.0814276933670044, 1.082715630531311, 0.3674046993255615, -1.4728610515594482, 0.5647547245025635, -0.6608186960220337, 0.36848118901252747, 1.357038140296936, 0.5691242218017578, -0.861664891242981, 2.4146831035614014, 0.006233966443687677, -0.7079377770423889, 0.4455125331878662, 0.509881854057312, 1.4249918460845947, 0.9763542413711548, -2.2146177291870117, 0.04089708626270294, 0.14159689843654633, -1.3107870817184448, 0.9821407198905945, -0.5253891348838806, 0.5572794079780579, -0.5630801916122437, 0.8258534073829651, 0.2757410407066345, -1.297735571861267, 0.3374515473842621, -0.7644073963165283, 0.4701266586780548, -0.06195346266031265, 0.046312566846609116, 0.38160112500190735, -0.6684589982032776, -0.06259424239397049, 0.20586098730564117, -0.4121834337711334, 0.03835328295826912, -1.1178257465362549, -0.38748082518577576, -1.504547357559204, 1.463090419769287, 0.5023552775382996, 1.152023434638977, -0.9152519106864929, -0.25018712878227234, 0.728015661239624, 0.7456325888633728, -1.0367743968963623, -1.5808948278427124, 1.0837087631225586, 0.5885595679283142, -0.20964252948760986, -1.2958654165267944, 0.11573994904756546, 0.3776843249797821, 0.17485353350639343, -1.1599773168563843, 1.4082505702972412, 0.7042921185493469, -0.6973488926887512, -0.44726064801216125, 0.35634204745292664, -0.30980589985847473, -0.6301432251930237, 0.30911123752593994, 0.15006926655769348, 0.08528252691030502, 0.41521838307380676, -0.5534296631813049, -1.5045056343078613, -0.2622453570365906, 0.12966449558734894, 0.13709311187267303, -0.1919209510087967, -1.684279441833496, 0.1315755993127823, -0.6951091289520264, -0.031362034380435944, 0.5543093681335449, -0.6975452899932861, 0.907555341720581, -0.9066988825798035, -0.24735963344573975, 1.4993211030960083, 0.05733766034245491, -0.3229176700115204, 2.1511785984039307, -0.3687944710254669, -0.3357747793197632, -0.5932119488716125, 0.6121166348457336, 1.9116127490997314, 0.7996270656585693, -0.22118864953517914, -0.6396692395210266, -0.12328256666660309, -0.7357857823371887, 0.6666836142539978, -1.2724303007125854, -1.484645128250122, 1.1970365047454834, -0.6484578847885132, 1.521133303642273, 0.7749916315078735, 0.7771819829940796, 0.11405688524246216, 0.513552188873291, -0.2661556005477905, -0.7587667107582092, -0.5905357003211975, 1.8773506879806519, -0.3304864168167114, -0.18075765669345856, -0.24676848948001862, -0.3534306585788727, 0.7765010595321655, 1.2589857578277588, -0.8519235253334045, 0.14420360326766968, 1.5217190980911255, 1.306183099746704, 0.35823729634284973, -0.48823264241218567, 0.4283175468444824, -0.6336294412612915, 1.3070881366729736, -0.41889962553977966, -0.8134503960609436, 1.8069173097610474, 1.415855884552002, 0.7961631417274475, 0.6258055567741394, -2.563934326171875, -0.8659797310829163, -0.5574161410331726, 0.8366692662239075, 1.5282328128814697, -1.447784423828125, -0.7917183041572571, -1.0810428857803345, 1.3883206844329834, -0.31919828057289124, 1.042155146598816, 0.13800357282161713, -2.5518460273742676, 0.3653820753097534, -1.0659571886062622, 0.6599301099777222, -0.9326881170272827, 0.21867461502552032, -2.383023500442505, -0.610460638999939, -0.00369556387886405, 0.7617232203483582, -0.4591309428215027, 0.3205125033855438, 0.23369485139846802, 0.18715298175811768, 0.40055468678474426, -1.9171966314315796, -1.0600606203079224, 0.2470601201057434, 0.9497388601303101, 0.4620909094810486, 0.7817252278327942, -0.1634647101163864, 1.7406874895095825, 0.7489656805992126, 0.16297659277915955, -0.9582120180130005, 0.3688380718231201, -0.6992099285125732, -1.593557357788086, 0.16808080673217773, -0.05704822391271591, 0.6229439377784729, -0.9744948744773865, 1.3547389507293701, 1.2219716310501099, -0.41468581557273865, -0.9399346113204956, -2.0910041332244873, 0.45455601811408997, 0.12221276015043259, -1.9262940883636475, 0.4398612678050995, -0.5852808356285095, -0.4696730971336365, 1.1769399642944336, -0.25049319863319397, 0.109258733689785, -0.08898469060659409, 0.45520198345184326, -0.2267664521932602, -2.241190195083618, 1.2247928380966187, 0.8435308933258057, -0.31771430373191833, 0.7664588093757629, -0.3353242576122284, 2.2759385108947754, 0.7322968244552612, -2.5687546730041504, 0.3347708284854889, 1.466105580329895, -0.48224952816963196, 0.7362020015716553, 0.3839666545391083, 1.0087604522705078, 0.39317187666893005, -0.5800238847732544, -0.436775803565979, 0.9014073014259338, 1.1291266679763794, 1.1099833250045776, 1.2042226791381836, -1.3463643789291382, 1.3809239864349365, 0.5544840097427368, -0.40598762035369873, -0.8213111758232117, -0.6623222231864929, 0.9167325496673584, -1.0050357580184937, 0.45070239901542664, -2.658848285675049, 1.1689354181289673, -0.26240137219429016, 0.3225560188293457, -0.09448016434907913, 0.3019254505634308, 0.7820125222206116, -1.1394689083099365, -2.2515578269958496, 0.24622690677642822, -1.2066028118133545, 0.09180304408073425, 0.6362820863723755, 0.5141396522521973, 0.29957520961761475, -0.37290987372398376, 0.12828433513641357, -2.340088367462158, 0.568710446357727, -0.5347630977630615, 0.598950207233429, 0.7755318284034729, 0.20697520673274994, 0.17302346229553223, 1.50741446018219, 0.0669097825884819, 0.6491283178329468, 1.0289809703826904, 0.5009112358093262, 0.10641622543334961, -2.26206636428833, 0.6268067359924316, -1.5023170709609985, -0.3196902573108673, 1.7369270324707031, -0.26051318645477295, 0.43709710240364075, -0.5721672177314758, -0.8197972178459167, -1.9049463272094727, -1.4220986366271973, -0.6867390275001526, 0.8066003918647766, 0.6774094104766846, 0.9043871760368347, -0.013882347382605076, 1.0123037099838257, 0.34694865345954895, 2.356125593185425, -0.5587829947471619, -0.6840122938156128, 0.9502406716346741, 1.270369052886963, -1.4546080827713013, -1.3200535774230957, 1.2949455976486206, -1.0690279006958008, -0.3069861829280853, 0.6475656628608704, -0.053628820925951004, 1.590490698814392, 1.1467375755310059, 1.5960005521774292, 0.22142019867897034, -0.43240323662757874, 0.11994682997465134, -0.03028201125562191, 0.3136650621891022, -0.5166782736778259, -1.379515528678894, 0.5047579407691956, -1.8039213418960571, 0.7828224301338196, -1.305905818939209, 1.3906232118606567, -1.130489468574524, -0.12841545045375824, 0.6269965171813965, 0.0960073173046112, -0.26210716366767883, 0.8759929537773132, 0.8511156439781189, 0.21980762481689453, -2.235595703125, -1.3253698348999023, -0.17550191283226013, 0.08814115822315216, -0.5532730221748352, 0.4335433542728424, 0.45181041955947876, 1.0631872415542603, 0.2792418599128723, -2.3108904361724854, 2.2973618507385254, -1.1407370567321777, -0.12100346386432648, -0.08514531701803207, 0.01460818201303482, -0.4693969786167145, -0.5324762463569641, 1.0829945802688599, 0.09500836580991745, -1.7013806104660034, 0.600228488445282, 1.0060423612594604, 1.5501329898834229, -0.16269473731517792, 0.7239723801612854, -1.2682892084121704, 0.6375724673271179, 1.2814618349075317, 1.2624775171279907, 3.177112579345703, -1.157108187675476, -0.20381899178028107, -0.8049142360687256, 0.8745551109313965, 1.1664484739303589, -1.6331170797348022, -0.8656166195869446, 0.13560621440410614, -1.2881946563720703, 3.6261613368988037, 0.4143546223640442, -1.0466758012771606, 0.4237750768661499, -0.6495575308799744, -1.724446177482605, -2.487656354904175, -0.05944911763072014, -1.1163690090179443, 0.04715407267212868, 0.13256028294563293, 1.019651174545288, 0.2840307354927063, -1.0187313556671143, 1.1829317808151245, -1.4181511402130127, 1.4401240348815918, -2.673872947692871, 0.17282555997371674, 0.06186724081635475, -0.06523256748914719, -0.7229012250900269, 0.3176816701889038, -1.1028227806091309, -2.412968158721924, 1.7938951253890991, -0.7113515734672546, 1.5737868547439575, -0.07352005690336227, -1.2077765464782715, 0.18097272515296936, -0.8147486448287964, -0.7495830059051514, 0.4698987603187561, 1.3495813608169556, -0.9900760054588318, -1.7777326107025146, 0.46350961923599243, -0.2020025998353958, -1.5854028463363647, -1.0001282691955566, 2.3887522220611572, -0.7951650619506836, 1.143318772315979, -0.36072489619255066, -0.4151051342487335, -0.9934328198432922, 1.0511772632598877, -0.11931493878364563, -1.4134247303009033, 0.661096453666687, 0.10896256566047668, -0.23038053512573242, 0.19415950775146484, -0.4011493921279907, -0.6863875985145569, -1.5153230428695679, -0.5370681285858154, 0.8679943680763245, -0.6535925269126892, -1.4478211402893066, 1.0295467376708984, 0.4111860394477844, -0.3595694899559021, -0.0128414211794734, -0.4296787679195404, -0.35508495569229126, -0.8510120511054993, -1.235758662223816, 2.0093865394592285, -0.5144383907318115, 0.028308499604463577, -1.1962049007415771, 0.7396440505981445, 1.2183575630187988, 1.0392166376113892, 1.4998936653137207, 0.9431968331336975, 0.5923951864242554, -1.0995150804519653, -0.9369855523109436, 0.0650368332862854, -0.37409695982933044, -0.4096122086048126, 0.26344382762908936, 0.06873318552970886, 2.0772643089294434, -0.2080778032541275, 0.08911842852830887, 2.3475730419158936, -0.8195876479148865, -0.01234694104641676, -0.9400999546051025, -0.790894627571106, -1.6159976720809937, 0.1145162507891655, 0.6183801293373108, 0.6954443454742432, -0.030284717679023743, 0.5888763070106506, -1.4444150924682617, -0.12841208279132843, 0.43024739623069763, 0.2003767341375351, -0.0943833440542221, 0.19549737870693207, 0.9514697194099426, -0.7836270332336426, 0.9808555245399475, -0.44773387908935547, -1.0841662883758545, -1.2502601146697998, -0.36412039399147034, -2.370870351791382, -1.5997368097305298, -0.8263546228408813, 1.1643693447113037, 0.8790980577468872, -0.5489372611045837, -0.39117100834846497, 0.011747152544558048, 0.19296130537986755, 1.272642970085144, 0.19296573102474213, -0.05522466078400612, 1.0885604619979858, -0.9882433414459229, -0.7397101521492004, -0.671249508857727, -0.24529516696929932, 0.6300138831138611, -0.27786099910736084, -0.5029821991920471, -0.015957636758685112, -0.01549191027879715, 0.352829247713089, -0.8879183530807495, -1.4913755655288696, -0.10733655095100403, 1.0420578718185425, 0.4701130986213684, 1.3191925287246704, 0.1201401948928833, 0.24464325606822968, 1.0420371294021606, 0.22892723977565765, 0.33518144488334656, 1.9731777906417847, 0.4686143696308136, 0.09846863150596619, -0.9979541897773743, -1.5185871124267578, 0.5824783444404602, 0.3383656442165375, 0.5707494020462036, -0.9274536371231079, -1.2458785772323608, 0.5164928436279297, 0.3870329260826111, 0.8305752277374268, 0.5066559314727783, -0.940515398979187, -0.15557126700878143, -0.275935560464859, 0.28874748945236206, -0.6071376800537109, 0.8463969826698303, 0.5016002058982849, 0.5024212598800659, -0.10780367255210876, -0.5295922756195068, 0.5137979388237, -1.0767790079116821, 2.622124433517456, 0.39851871132850647, 0.2819684147834778, -0.8090013265609741, 0.11803580075502396, -0.14049981534481049, -2.2221758365631104, 0.1251882016658783, -0.8874131441116333, -1.6331459283828735, 0.8540909290313721, -2.765073537826538, -0.37655165791511536, -1.3941223621368408, -0.851060688495636, 0.48811718821525574, -0.33068686723709106, -0.14603865146636963, -0.15488609671592712, 0.4543660283088684, 2.2670793533325195, 0.2923828363418579, 0.28480228781700134, -0.1547899693250656, -0.932057797908783, -1.0672519207000732, -0.4553307890892029, 1.056004524230957, -0.9858867526054382, 1.6949537992477417, -1.836783766746521, -0.38335487246513367, 0.33584970235824585, 0.2804146409034729, -0.4763319492340088, 1.101137399673462, 1.0763599872589111, -0.1526695340871811, -0.025915250182151794, -0.4429560601711273, 0.4619303047657013, -1.0194272994995117, 0.08531332015991211, 0.051436737179756165, -1.0507019758224487, 0.8586632609367371, 1.4549310207366943, 1.599056363105774, -0.1831320971250534, 1.1261183023452759, -1.4049313068389893, 1.1748754978179932, 1.4985450506210327, 0.10668922960758209, -1.0125534534454346, -0.07377360761165619, -0.5092618465423584, 0.6861897706985474, 1.0775736570358276, -1.3927712440490723, 1.447413682937622, 0.8130297064781189, 1.2319267988204956, -1.5515519380569458, -1.4011212587356567, 0.277195543050766, 1.373246192932129, 1.2812844514846802, -1.6420388221740723, -0.4264408051967621, -0.6593143343925476, -0.23918327689170837, 2.051201343536377, -0.044302258640527725, 0.6612332463264465, 0.8091723322868347, 0.378579705953598, 0.5809643268585205, 0.3694697618484497, -0.5048320293426514, 1.6359336376190186, 0.6939358115196228, 0.911410927772522, -0.497903048992157, -0.4216311573982239, -0.8657857775688171, 0.12254100292921066, -0.19229206442832947, -0.6277162432670593, 1.0063154697418213, -0.30458393692970276, -0.18749766051769257, -0.21989493072032928, 0.8738687634468079, -0.07391411066055298, 0.126047283411026, -1.954853892326355, -0.5259642004966736, 0.29649391770362854, -0.4782352149486542, -1.5595532655715942, -0.5055409073829651, -0.4694010019302368, -3.101741075515747, 0.8670752048492432, -0.5904134511947632, -0.6787434220314026, 0.7293028235435486, 0.31963595747947693, -0.26336613297462463, 0.09633415937423706, -2.1334033012390137, 1.5630685091018677, -1.1362078189849854, -0.943566620349884, 0.16125278174877167, 0.5411932468414307, 0.4449596107006073, 0.7134475708007812, -1.2862330675125122, 0.2629663050174713, 0.13654713332653046, -2.2293879985809326, -0.6730234026908875, -0.5982103943824768, -1.3187806606292725, -1.2778481245040894, -0.3431415557861328, 1.063123345375061, -0.4223734438419342, -0.700042724609375, -0.3202368915081024, -0.31321263313293457, 0.4804975688457489, -0.7686516642570496, -0.9843543767929077, -1.3265314102172852, -0.4102482199668884, 0.5626603364944458, -1.7679574489593506, 0.023019950836896896, -0.8170472383499146, -0.07890872657299042, -0.18639720976352692], [-1.5350500345230103, -0.12411773204803467, 0.20666062831878662, -0.6801348328590393, -0.4866466224193573, 1.6791749000549316, -0.7015308737754822, 0.34834542870521545, 0.38311317563056946, -1.1585333347320557, 2.249925136566162, 3.162851572036743, -1.485525369644165, 2.3500754833221436, -1.8787250518798828, -0.2642865478992462, -1.5332388877868652, 0.4314475953578949, -0.6251056790351868, -0.9703997373580933, -0.16817624866962433, -0.6577783823013306, -0.4243153929710388, -0.31958967447280884, 0.1420506089925766, 1.5556068420410156, -0.7740083932876587, 0.11885159462690353, -0.1868056058883667, 1.254921793937683, 0.4831986129283905, 2.469013214111328, 0.6545912623405457, -0.010443467646837234, 0.6795768737792969, -0.14522768557071686, -0.6908383965492249, -0.5576210618019104, -0.623857855796814, -0.29900169372558594, 0.11662403494119644, 0.05833110213279724, 0.4939456582069397, 0.7147135734558105, 0.7842366099357605, 0.22463814914226532, 0.4860065281391144, 0.7940298318862915, -0.4599693715572357, -1.6342313289642334, -0.6667875051498413, -0.2610628604888916, 0.34914708137512207, 1.0480338335037231, -0.6248814463615417, -0.7721213698387146, -0.12279213964939117, 0.11990583688020706, 0.14682871103286743, 0.2829688489437103, -0.03098512440919876, -2.2868504524230957, -1.4528089761734009, -1.2216838598251343, -0.10680294781923294, 1.5555298328399658, 1.3019896745681763, -0.14629559218883514, -0.07074424624443054, 0.23888032138347626, 0.7445966601371765, -0.0608070082962513, -0.004483389668166637, -0.5758222937583923, 0.5256110429763794, -1.3741300106048584, -0.28572067618370056, 0.150024875998497, -0.6787486672401428, 1.5349808931350708, -0.05616581439971924, 1.177701711654663, 1.5690096616744995, 1.1605939865112305, -0.8822697997093201, -0.6798827648162842, -0.14192864298820496, 0.1532556563615799, -1.4171758890151978, 1.8980003595352173, 0.07223353534936905, -1.238061785697937, 1.0679157972335815, -0.24210433661937714, -1.6337908506393433, -1.4087300300598145, -0.3956318199634552, -0.40863946080207825, -0.21676678955554962, 0.12350870668888092, -1.616862177848816, 1.0799565315246582, -1.8289456367492676, 1.4140747785568237, 0.49000051617622375, -0.7530428767204285, 0.9503013491630554, -0.2339809089899063, -2.108630418777466, 0.4958009719848633, 1.663521647453308, -2.252547025680542, 0.08840341120958328, 0.3277265131473541, 1.0407894849777222, 0.8291078209877014, 0.15271809697151184, -0.6552734971046448, -0.6022956371307373, -0.01612081006169319, 0.47150978446006775, -1.3315571546554565, 0.07024755328893661, -0.04214927554130554, 0.3374393582344055, -0.36040806770324707, -1.0138782262802124, 0.895456850528717, -0.20498262345790863, 0.5577179789543152, -0.9518853425979614, 0.004680209793150425, 0.3761169910430908, -0.8913094997406006, -2.1987264156341553, -2.264962911605835, -1.086739420890808, 0.600311279296875, 0.6561511754989624, 0.7157822847366333, 0.6475275754928589, 0.08866661041975021, -0.4662139117717743, -0.8656078577041626, -0.4503982365131378, -0.475251704454422, 0.07363743335008621, 1.8216032981872559, -1.6719212532043457, -0.3164786994457245, -1.3552864789962769, 0.35983988642692566, 0.34878087043762207, 1.1553747653961182, -0.4885326623916626, 0.6667914986610413, -1.5461866855621338, -0.5093755125999451, 0.3235742747783661, -0.9700220823287964, -0.49511608481407166, 0.5739504098892212, 0.4948752224445343, -0.7977347373962402, -1.4193164110183716, -0.08434994518756866, -0.3477184474468231, 1.1326643228530884, 0.21596068143844604, -0.3531496822834015, -0.9554058313369751, 0.29845625162124634, -0.233362078666687, 0.12020434439182281, 0.4963848888874054, -0.8705440163612366, -0.17359137535095215, 0.6283302307128906, 0.2506750226020813, 0.3668082356452942, -0.21462929248809814, 1.7513165473937988, -0.551835834980011, 0.07613977044820786, -0.6140117049217224, 0.34748193621635437, 0.32070520520210266, 0.0604630783200264, -0.7635586857795715, -0.047242697328329086, -0.024493254721164703, 0.016025139018893242, -0.3814169466495514, -1.3000015020370483, -1.3617370128631592, 0.045565616339445114, -1.6141425371170044, -0.4264422357082367, 1.1616199016571045, -1.5097935199737549, 0.48281973600387573, 1.4996464252471924, -0.6850088238716125, -0.9462078213691711, -0.45300257205963135, 0.6075311303138733, 0.7136343717575073, -0.13358937203884125, -0.683106541633606, -0.3953694701194763, -1.1506315469741821, -0.48073408007621765, -0.043083421885967255, 0.16368383169174194, 1.0143303871154785, 1.0850378274917603, 0.6744441390037537, 1.011077880859375, 0.7546983957290649, 0.455976277589798, 1.082980751991272, 1.0254333019256592, 0.35690444707870483, -0.9160115122795105, -1.3459056615829468, 0.18200340867042542, -0.7342079281806946, 0.3569994270801544, 0.9998306035995483, -1.188829779624939, -0.9314331412315369, 0.6956940293312073, 1.0687787532806396, 1.05645751953125, -0.6174423694610596, 0.3735686242580414, -1.0469646453857422, 0.21132564544677734, -0.8811867833137512, -0.9895613789558411, 0.5865887403488159, -0.19866222143173218, 0.40497487783432007, 0.06263960152864456, -2.9052186012268066, 0.4632014036178589, -0.11983102560043335, -1.0288150310516357, 0.5305494666099548, 1.1949527263641357, 0.485592782497406, 1.4847493171691895, -0.4530050456523895, -0.9682918787002563, -0.14136147499084473, 0.11708205938339233, 2.178318977355957, 0.9416211843490601, 1.4251272678375244, 0.7203773260116577, 0.8204735517501831, -0.5760961771011353, 0.9251714944839478, 0.27624741196632385, -0.6925164461135864, -1.9312623739242554, -1.1771312952041626, -0.20572754740715027, -1.0010054111480713, -0.7080963850021362, -0.58525550365448, 1.1421270370483398, 0.949294924736023, -0.9965548515319824, -1.2912859916687012, 0.5707347989082336, -0.4586034417152405, -0.08208853751420975, 0.20854723453521729, -1.552642822265625, 0.009728219360113144, 2.2848305702209473, 0.48073065280914307, 0.9302528500556946, 1.0916028022766113, 0.5160188674926758, 0.469834566116333, -2.2116849422454834, -0.9504870772361755, -3.3781394958496094, -0.1956896185874939, -0.2899773418903351, -0.12087512761354446, 0.6325293183326721, -1.270097017288208, 0.6789135932922363, 0.6788926124572754, 0.7225148677825928, 0.04730810970067978, 0.09147275984287262, -0.7518805265426636, 0.12503491342067719, -0.007808785419911146, 0.35800760984420776, 0.3247893750667572, -0.12235138565301895, -0.29666662216186523, 0.49154233932495117, -0.6869570016860962, 2.644015073776245, 0.6258463263511658, -1.148597240447998, 1.1728310585021973, -1.2369736433029175, 0.4117768406867981, 0.20637015998363495, 1.0885813236236572, 1.7188851833343506, -0.16555413603782654, 0.08900607377290726, 1.1950267553329468, -1.4185535907745361, -0.42591729760169983, 2.0199637413024902, -1.1062030792236328, -0.4846823513507843, 1.322754144668579, -0.018198320642113686, -0.8671687841415405, 1.4807713031768799, 1.6674412488937378, 0.219825878739357, -1.5330415964126587, -1.4701584577560425, -0.8790934681892395, -0.12105391919612885, -1.8729584217071533, 0.018749358132481575, 1.0042715072631836, 1.2165677547454834, -1.1543673276901245, -2.006967067718506, -0.5628769993782043, -0.2454833984375, 0.08764418214559555, -0.2346632480621338, 0.37355896830558777, -1.205475091934204, -2.710413932800293, 0.2586379945278168, 0.7653732895851135, -1.0258500576019287, 1.5650404691696167, 0.20361852645874023, 0.6794900298118591, -0.09355174750089645, 0.20305724442005157, -1.7828553915023804, 0.4144325852394104, 0.6962395906448364, -1.0172330141067505, -0.399577260017395, 1.6544748544692993, 0.8009609580039978, 0.808585524559021, -0.2523936331272125, -2.5140364170074463, 0.08455070108175278, 0.6807887554168701, -0.7096276879310608, -0.7418349981307983, -0.35697266459465027, 0.5709337592124939, -1.3112590312957764, -0.4656396508216858, 0.0913129672408104, -0.3611794114112854, -0.8037528395652771, 0.8476049304008484, -0.04216836392879486, 0.8397170901298523, 0.8186324238777161, -0.052873894572257996, 2.4044995307922363, 0.9608368277549744, 0.13961781561374664, -0.15668989717960358, -1.1041855812072754, 0.6501603722572327, -0.7003160715103149, 0.9545139074325562, -0.15559986233711243, 0.9813178181648254, 0.6207389831542969, -0.19926978647708893, -0.9471064209938049, 2.2289464473724365, -0.6543604731559753, -0.7089412808418274, -0.9038054943084717, -0.5862353444099426, 1.3465492725372314, 0.3505221903324127, 1.4587558507919312, 0.801044762134552, 0.7750253677368164, -0.2281646579504013, -1.3113949298858643, -0.4038480520248413, -0.2017299085855484, -1.5455009937286377, 0.6169866919517517, 0.0014417065540328622, 1.071399211883545, 0.40918076038360596, -0.6065832376480103, 1.017116904258728, -0.08540742844343185, -0.08288680762052536, 0.7598509192466736, 0.5366308093070984, 0.8580979108810425, 0.8023574948310852, -0.571080207824707, -0.8067523241043091, 0.9564064145088196, -0.865896463394165, -0.4214407503604889, 0.8947845101356506, 1.765395998954773, -1.228805661201477, 0.9736856818199158, -0.4347672462463379, 1.6673822402954102, 1.0249489545822144, 3.5075864791870117, -0.9362865686416626, 0.2367812544107437, -0.5855451226234436, 1.7404125928878784, -1.5017777681350708, 0.7132202982902527, -2.32570219039917, 1.5184097290039062, 1.154211401939392, 0.23856766521930695, 0.11456213891506195, 0.24868132174015045, -0.05527743697166443, -0.5234904885292053, 0.21454884111881256, -2.1981682777404785, -1.318699598312378, -1.2604233026504517, 1.135535478591919, 1.9089475870132446, 0.7944458723068237, -0.4058319926261902, 0.9652804732322693, -0.8981212973594666, -0.18708841502666473, -0.3364187180995941, 1.0302820205688477, -0.26645559072494507, 2.847562551498413, 1.3028955459594727, 0.7026647925376892, 0.8866478800773621, -0.41429024934768677, -0.045411113649606705, 0.7997171878814697, -0.9254217743873596, 0.340232253074646, -0.9260298609733582, 0.04114565625786781, -0.45788174867630005, 0.21330484747886658, -0.09421740472316742, 1.5163908004760742, -0.05161141976714134, -0.5204238891601562, -0.2636652886867523, -0.24975810945034027, 1.2641066312789917, 0.43488502502441406, 0.48298904299736023, -0.6350493431091309, 0.4412011504173279, -1.4242368936538696, 1.2404576539993286, 0.09751633554697037, 0.5079941153526306, -0.02470589615404606, -0.1786404401063919, 1.1790224313735962, -0.5046349763870239, 0.4277587831020355, -0.4913206398487091, -2.1311821937561035, -1.5179417133331299, 1.132877230644226, 0.5733721852302551, -1.0178364515304565, -0.20633618533611298, 0.9004549384117126, -1.56831955909729, 0.5424476265907288, 0.4524766802787781, -1.3390297889709473, 1.2378969192504883, -0.8412607312202454, -0.9738417267799377, -0.9166328310966492, 1.720035195350647, -0.4540567398071289, 0.009598910808563232, -1.118247628211975, 0.10827978700399399, 0.9837045073509216, 0.15202948451042175, 0.2203543484210968, -0.6414125561714172, 0.7099510431289673, -0.11130648106336594, 0.660585880279541, 1.1616780757904053, -1.5163319110870361, 0.7242410182952881, -1.7381588220596313, 0.2275911569595337, 1.8797497749328613, 0.263440877199173, -0.6412730813026428, 0.48748981952667236, -0.5933583974838257, -0.49822160601615906, 0.8020021319389343, -0.23830518126487732, 1.419329285621643, 0.39011150598526, -0.5031707882881165, 0.4804832935333252, 0.9047242999076843, 0.31923967599868774, -0.5447224974632263, 0.2676393985748291, -0.6955592632293701, 0.4772660732269287, 0.15211080014705658, 0.0253083948045969, -1.2382258176803589, 0.3863748610019684, -0.10749020427465439, 0.3132210969924927, 1.057963490486145, -0.5118452310562134, -0.32082659006118774, -0.06105343624949455, -1.8213675022125244, 0.2507355511188507, -0.589297354221344, 0.03915006294846535, -0.006982584949582815, -0.38343608379364014, -2.016289472579956, 1.810081124305725, -0.5284612774848938, 1.293775200843811, -0.7477356791496277, 0.3734554052352905, -0.5356730222702026, 0.6682156920433044, -0.362658828496933, 1.1841702461242676, -0.38906311988830566, 0.35340407490730286, 0.42978909611701965, 0.8512558937072754, 1.560097098350525, 1.1056346893310547, 0.3038821220397949, -0.28902530670166016, 0.4586273431777954, -0.10038244724273682, -1.3081274032592773, 0.13231872022151947, -0.1609572321176529, 1.371017575263977, -0.23920927941799164, 0.4084922671318054, 0.8984359502792358, -0.5110050439834595, 0.5841044187545776, -0.6862457394599915, -2.2704317569732666, 0.19125452637672424, 0.69312584400177, 0.5333226919174194, -0.24689249694347382, -0.6649564504623413, -0.020142100751399994, 0.9780579209327698, -1.2488527297973633, -0.9388542771339417, 1.638123631477356, 0.839651882648468, 1.0779905319213867, 0.24925056099891663, 0.8539683222770691, 0.45080888271331787, -1.1007691621780396, -0.7629530429840088, 0.11143261194229126, -1.111158847808838, -0.8729524612426758, 1.0647236108779907, 2.0789687633514404, 0.97446608543396, -0.8418574333190918, -0.25662970542907715, 1.6198506355285645, -0.8883118033409119, -0.027570180594921112, 0.8925462961196899, 1.4267802238464355, 0.23068757355213165, -2.1079273223876953, 1.0458903312683105, 0.6336494088172913, 1.1027246713638306, -0.9566428065299988, -0.35728421807289124, -0.2410183995962143, 0.5297544002532959, -1.9418463706970215, -0.8134329915046692, -0.007624190766364336, 0.7055369019508362, -0.260553240776062, -0.5238732695579529, -0.16130635142326355, -0.5438584089279175, -0.40542370080947876, -0.06814350187778473, 0.9245288968086243, -0.06593936681747437, 0.7552682757377625, 0.8908252120018005, -0.4262016713619232, -1.0078849792480469, -1.062607765197754, 0.49753570556640625, 0.5984826683998108, 1.5318174362182617, -0.061051592230796814, -1.1159064769744873, 0.6594451069831848, -0.6671274304389954, 1.7056535482406616, 1.9746067523956299, 0.15950123965740204, 1.2276086807250977, 1.4438014030456543, -0.07437973469495773, 0.3607783913612366, 0.7044075131416321, 0.32618504762649536, 0.4828723967075348, -0.5853889584541321, 0.7120200395584106, -1.5067312717437744, 0.6972506046295166, 0.12757234275341034, 1.8032573461532593, 1.0250223875045776, -1.918747901916504, 0.3810771405696869, -1.47262704372406, -0.0697043389081955, 0.2520272135734558, 0.2551186978816986, 0.049099069088697433, -0.378158301115036, -0.8621163964271545, -0.04322643205523491, 1.0249887704849243, -0.5661621689796448, -1.173509955406189, -0.5501309037208557, 0.22952280938625336, -0.9945173263549805, 0.19473369419574738, -0.7775338888168335, -2.4160773754119873, 0.5218125581741333, 0.44491520524024963, -0.32191044092178345, 1.779631495475769, -0.1832152009010315, -1.8041601181030273, 0.1713339239358902, -1.3197332620620728, -1.2372618913650513, -0.40903928875923157, -1.7005606889724731, 1.0885039567947388, 0.6344516277313232, 1.3071503639221191, -0.6459970474243164, 0.7343868613243103, -0.42184582352638245, -0.7436302304267883, 0.31476065516471863, 1.2840176820755005, -0.2769930958747864, -0.771129846572876, -0.7323200702667236, -1.0066478252410889, -0.3714433014392853, 0.33987346291542053, -0.9488852024078369, 1.3677735328674316, -0.6881663799285889, 1.42559015750885, 0.8154873847961426, 1.3516721725463867, -0.7931304574012756, 0.4560793340206146, 0.29118579626083374, -2.2676334381103516, -0.13327136635780334, -1.6136705875396729, 1.454447627067566, -0.4630033075809479, 0.6401742100715637, 0.047732461243867874, 0.3522551357746124, -0.6027671098709106, -0.9875104427337646, 1.2166777849197388, 0.371133029460907, -0.5851837396621704, -0.33449676632881165, -0.21672968566417694, -0.6796550154685974, -0.47659531235694885, 0.471047580242157, 1.161679983139038, 0.5247563123703003, -0.9447665810585022, 0.0065570673905313015, -0.8687002658843994, 0.8981428742408752, 0.21046140789985657, 0.42840149998664856, -0.7539588809013367, 1.2652645111083984, 0.4709133803844452, -0.07671791315078735, -0.5741325616836548, -0.9886912107467651, 1.447981834411621, 0.06453026831150055, 1.45246160030365, 0.3047565519809723, 1.3970788717269897, 0.76358962059021, -0.48098912835121155, -0.25027695298194885, -0.4116978645324707, 0.14436766505241394, -0.9637688398361206, -1.6732878684997559, 0.37335583567619324, 1.127345085144043, 0.42664071917533875, -0.5430410504341125, -0.20432795584201813, -0.8193287253379822, 0.539262592792511, -0.10489953309297562, 0.7320889234542847, -0.4629884660243988, -2.8597943782806396, -0.4668472707271576, -1.004068374633789, -0.8286539912223816, 1.356584906578064, -0.07930400222539902, 0.4918733835220337, 0.35508015751838684, 0.16711053252220154, -0.3845210373401642, 1.0512129068374634, 0.8694947361946106, -1.0587198734283447, -1.3846782445907593, -0.26425760984420776, -1.3989863395690918, -0.4480227828025818, -0.24270866811275482, -1.0803730487823486, -0.28315743803977966, -0.7411177158355713, 0.4033201336860657, -0.9850305318832397, 0.2099471092224121, -0.2633771598339081, -0.9992108345031738, 0.6955345869064331, 0.23527120053768158, 0.12949497997760773, 0.7370783090591431, 0.3229442834854126, 0.18909168243408203, 1.1865358352661133, -1.8999770879745483, -0.9599414467811584, -0.13061460852622986, -0.33637070655822754, 0.9455634355545044, -0.5041905641555786, 1.5636637210845947, 0.8763964772224426, 0.734336793422699, 1.8256944417953491, -0.04450184479355812, -1.369513988494873, 0.3914373517036438, -0.20863856375217438, -1.006325602531433, 1.0028177499771118, -0.04518267139792442, -1.0339709520339966, -1.7879540920257568, -0.12453947961330414, 0.37142664194107056, -1.6281492710113525, 0.0935106873512268, 1.067879319190979, -1.8216782808303833, 0.4244978725910187, 0.30703091621398926, 0.5174571871757507, -0.8356582522392273, -1.2467046976089478, 0.654944658279419, -0.5963132977485657, 1.1611229181289673, -1.5461149215698242, 0.11502912640571594, -0.27478134632110596, -0.7765883207321167, -0.17346012592315674, 1.2563308477401733, 0.07196887582540512, -0.10146915167570114, -0.4313879907131195, 1.096024990081787, 1.2027262449264526, 0.9544145464897156, 1.19217050075531, -1.94089937210083, -0.3686319887638092, 1.428175926208496, 0.5330050587654114, -0.3218385875225067, -0.9751418828964233, -0.07979217171669006, -1.879590630531311, -0.3521479070186615, -2.6968867778778076, -0.12306128442287445, 0.3048732578754425, -0.19558273255825043, -0.005009880755096674, 1.9667412042617798, 1.7232098579406738, -0.4335253834724426, -1.0954185724258423, 1.6304203271865845, 0.8989214897155762, 0.8700862526893616, -0.06526733189821243, 0.28082647919654846, 0.42343980073928833, 0.7127630710601807, 0.22771498560905457, -0.789353609085083, -0.670944094657898, 0.027044864371418953, 1.3925466537475586, 0.2698976695537567, 0.5420010685920715, -1.3879057168960571, -0.0489753782749176, 0.2918283939361572, -0.13111168146133423, 3.614600419998169, -0.5760050415992737, 0.21986089646816254, -1.1107984781265259, -1.9855543375015259, 0.7267389297485352, -0.07871034741401672, 0.24943087995052338, 0.6100653409957886, -0.9512282609939575, -1.7906761169433594, 0.4529738426208496, -0.10820358991622925, 0.6993860006332397, -1.03223717212677, -0.0008195704431273043, 0.7376255989074707, -0.20035235583782196, 0.0640852302312851, 0.8523107767105103, -0.07636059820652008, 0.007328465580940247, 0.1371394693851471, -0.7735123038291931, -0.5068768262863159, 0.67242032289505, -1.35865318775177, -2.380673885345459, -0.1530294269323349, 0.5918200016021729, -1.356937050819397, -0.7423610091209412, 2.0662803649902344, 0.5717200040817261, -0.7818101644515991, 0.00361488270573318, 0.6668469309806824, 0.03228169307112694, 0.15976710617542267, 1.510077953338623, -0.6402250528335571, -1.3045116662979126, 1.4588303565979004, -1.0407887697219849, 0.05223941430449486, 1.2440768480300903, 0.5625087022781372, 0.5984241366386414, 0.5040316581726074, 0.5233370065689087, 1.1320387125015259, -0.42695754766464233, 2.0494422912597656, 0.03396128863096237, 0.07105729728937149, 1.0179353952407837, -0.9035055041313171, 0.30084845423698425, 0.2556839883327484, 0.5776695013046265, -0.9230110049247742, -0.5325692892074585, -0.9284870028495789, 0.676397979259491, 1.2384768724441528, -2.1268560886383057, -1.2429720163345337, 0.5965707302093506, 1.7688125371932983, 0.10724463313817978, 0.2687155306339264, -0.3403129577636719, 1.9407485723495483, -0.5414865612983704, 0.9246677160263062, 0.25278428196907043, 1.0676002502441406, -0.507453978061676, -0.20229464769363403, -0.3027900457382202, 0.03258566930890083, 0.8573679327964783, 1.478042483329773, 1.0451997518539429, -0.743459165096283, -1.6297476291656494, -0.22632195055484772, 1.7770172357559204, -0.7851393222808838, -1.3069723844528198, -0.6920758485794067, 0.881163477897644, -0.19125835597515106, -1.2289665937423706, -0.13748995959758759, 0.2867271900177002, 0.12374352663755417, -1.5055444240570068, 0.5597119331359863, 0.6545113325119019, 0.47867321968078613, -0.47683706879615784, -1.1579972505569458, -0.4061700999736786, -1.142254114151001, -0.7921521663665771, 1.6723078489303589, -0.324316143989563, 1.7935740947723389, 0.1342252939939499, -1.0301545858383179, 0.794908881187439, 1.1348973512649536, 2.377336025238037, -1.627535343170166, 1.0360466241836548, -0.4368550181388855, -0.33181893825531006, -1.1044694185256958, 0.35162657499313354, -0.01380245666950941, 0.027968866750597954, -0.832187294960022, 0.6552417278289795, -1.2659164667129517, 0.09225089848041534], [0.8019302487373352, -1.253678798675537, -0.9702175259590149, 0.8883497714996338, 0.30373865365982056, 1.1902271509170532, -3.0454535484313965, -0.7599038481712341, 2.067502975463867, 0.7039926648139954, 0.6078060269355774, 1.0335197448730469, 0.030366653576493263, 0.5915214419364929, -1.2463836669921875, -0.019154783338308334, 0.7798212766647339, -2.2534677982330322, 0.017063379287719727, 0.44311484694480896, 1.0604597330093384, -1.636443018913269, 0.40268203616142273, -0.017318332567811012, -0.5224848389625549, 0.622516930103302, 0.564978301525116, -0.17417046427726746, -2.112154006958008, 0.41897207498550415, 0.603130042552948, -0.6428071856498718, -0.5836424231529236, -0.15472979843616486, 0.9720973372459412, -0.2199118584394455, -1.3276559114456177, 0.6654915809631348, -0.7645184993743896, 0.7122417092323303, 0.16150707006454468, 0.6753328442573547, 0.1796545684337616, -1.0775498151779175, 0.9160758256912231, -1.6842246055603027, -0.11892251670360565, 1.4516106843948364, 0.07135021686553955, -0.31206393241882324, -1.343153953552246, 0.5722634792327881, 0.6335583329200745, 0.8854833841323853, 1.424825668334961, -0.8408093452453613, 1.690680742263794, -2.269090414047241, -0.17682860791683197, -1.428650975227356, -0.17965207993984222, -0.18501515686511993, 1.5932350158691406, -1.6171152591705322, 1.0442512035369873, 0.720556378364563, -0.4113214313983917, 1.918076753616333, -0.16666163504123688, 0.2920007109642029, 1.1379069089889526, -1.1538835763931274, 1.0821115970611572, -0.09823714196681976, 0.09132976830005646, -0.6742810010910034, -0.6801742911338806, 0.5479021668434143, 0.5750994086265564, -0.20557630062103271, 0.1264190673828125, -0.44567614793777466, 0.5313853621482849, -0.48804613947868347, -1.1957182884216309, 0.623092532157898, 0.6300929188728333, 0.8707635998725891, 0.2733307480812073, -1.5991771221160889, -0.8957189917564392, -0.4844626784324646, -0.4608682096004486, -0.25606414675712585, -0.3338160514831543, -0.46295079588890076, 2.160250663757324, 1.0708898305892944, -0.6431612372398376, -0.06866868585348129, -0.9960171580314636, 1.4210036993026733, -0.2960597574710846, 0.7132043242454529, 0.5358911752700806, 0.04634157940745354, -1.3750215768814087, -0.648560106754303, -1.0233384370803833, 0.17757926881313324, 0.3189653158187866, -0.27805933356285095, 1.2492318153381348, 0.5387306809425354, 0.07482409477233887, 0.5367924571037292, 0.7625724077224731, -1.3375989198684692, -0.7540987133979797, 1.3409847021102905, -0.29082706570625305, -0.9842816591262817, 1.8723548650741577, -0.8891279101371765, 2.818850040435791, 0.563616931438446, -1.1298432350158691, -1.4459171295166016, 0.07170300930738449, -0.3218788504600525, -2.9183478355407715, 0.694060206413269, 0.4113207459449768, 0.6331564784049988, 1.1558054685592651, -1.323983907699585, -0.9638956785202026, 0.25992080569267273, -0.938778817653656, -0.22486016154289246, -0.023237982764840126, -0.6662126779556274, 0.3835328221321106, 0.2682260274887085, 0.01682465150952339, -0.5000573992729187, 1.0304932594299316, 1.328953742980957, 0.5466110706329346, 0.43473684787750244, -0.5567174553871155, -1.141730785369873, 0.44622400403022766, 0.4273862838745117, -0.9239190220832825, -0.10205662995576859, -1.5118588209152222, 0.40945035219192505, -0.5824885964393616, 1.2117149829864502, 0.04653475433588028, 1.01560640335083, -0.6350232362747192, -0.17320159077644348, -1.232779860496521, -1.1951961517333984, -0.3719179332256317, -1.069493055343628, 2.289337396621704, 0.926236629486084, -2.312051773071289, -0.8301681876182556, -0.01942547969520092, 0.25212913751602173, 0.6567502021789551, 1.4458204507827759, -1.0301601886749268, -1.3031716346740723, -0.3250493109226227, 1.082374095916748, -1.8409310579299927, -0.48515936732292175, -2.80432391166687, 0.21614594757556915, 1.7369617223739624, 0.1705014407634735, -0.17875699698925018, -2.7712552547454834, 1.1307263374328613, -1.1910736560821533, -0.6040542125701904, -0.7156484127044678, 0.7094322443008423, 0.5918540954589844, 0.26550424098968506, 0.5390614867210388, 0.9812711477279663, -0.885806679725647, -2.1867306232452393, 0.15739373862743378, -0.031468816101551056, -0.035679735243320465, 1.020216703414917, -0.7263044714927673, 2.218721628189087, -0.9573805928230286, -0.07069174945354462, -1.8936172723770142, -0.6657535433769226, 1.1742630004882812, 0.6558502912521362, -2.273916244506836, -0.271118700504303, -1.140278935432434, 0.19270910322666168, -0.36688166856765747, -0.02752680703997612, -0.9436011910438538, 1.8161553144454956, -0.45487523078918457, 0.03911321982741356, 0.14439207315444946, -2.048387289047241, 0.5054996609687805, -0.19730040431022644, -0.28539881110191345, 0.7079464793205261, 0.05299322307109833, -0.012723429128527641, -1.3732242584228516, 1.2034937143325806, 1.0745688676834106, -1.7064788341522217, 0.9232664704322815, 0.21083690226078033, -1.24982750415802, 0.8286585211753845, 1.9085298776626587, 0.12171588838100433, 1.3058514595031738, 0.31099197268486023, 1.0356823205947876, -0.36508336663246155, 0.6449294686317444, -1.920067548751831, 0.1096072793006897, -0.25445929169654846, 0.3964661657810211, -1.6590549945831299, 0.5331611037254333, -0.2952510118484497, -1.3883874416351318, 0.014192522503435612, -2.4817044734954834, -0.46122506260871887, 0.7404554486274719, 0.9528335928916931, -0.492171049118042, -0.523852527141571, -0.31213122606277466, -1.2343839406967163, -0.5556651949882507, -0.5692070126533508, 0.5867058634757996, -0.724556028842926, 1.637152075767517, 0.4131249189376831, 0.43291041254997253, -1.3266115188598633, 0.26937490701675415, -1.1299831867218018, -0.16842184960842133, 1.9166661500930786, 1.6854206323623657, -0.39317646622657776, -0.6948722004890442, 1.3515264987945557, -1.7976813316345215, -0.07883839309215546, -1.0901185274124146, 0.03986315429210663, 0.35010218620300293, 0.8278615474700928, -1.1260923147201538, -0.45571818947792053, -0.29009974002838135, 0.5027835965156555, -0.45388075709342957, 0.9245509505271912, 0.023595185950398445, 1.4063249826431274, -0.297308087348938, 0.7496342062950134, 0.4133385717868805, -1.1284184455871582, 1.1025418043136597, -0.2636672556400299, -1.2139490842819214, 0.7503425478935242, 1.0366917848587036, -1.5219850540161133, 1.8938285112380981, 0.32297295331954956, 0.0870426669716835, -0.6531174778938293, 0.5744140148162842, -0.8454299569129944, -0.004146887455135584, 1.1555036306381226, 0.25059279799461365, -1.0134750604629517, 0.09106138348579407, 0.7285997867584229, 0.6818721294403076, -0.9142818450927734, -0.6185315847396851, 1.5989278554916382, -0.8652723431587219, -0.45476147532463074, 0.6002418398857117, -1.5211784839630127, 1.1338857412338257, -1.8245469331741333, -2.1134917736053467, 0.18163566291332245, -0.4694644510746002, -0.9097346067428589, 1.1749835014343262, 1.3310967683792114, -0.2312350869178772, 0.10272543132305145, 1.4938182830810547, 0.8572396039962769, 0.7497699856758118, 1.1798858642578125, 1.2444794178009033, -0.6224112510681152, 0.7004927396774292, 0.117889404296875, 0.3484669327735901, 1.0427969694137573, -0.9433602094650269, -1.7720950841903687, -0.6763721704483032, 1.180864691734314, -2.6405625343322754, 2.1718571186065674, -2.06504225730896, -0.5881728529930115, 0.20476067066192627, 0.3256852924823761, -0.65450519323349, 1.1344717741012573, 0.24898488819599152, 0.24214541912078857, 0.40486934781074524, 0.5552997589111328, 0.14303290843963623, 2.3471763134002686, 0.9192757606506348, -0.45032230019569397, -0.6339778900146484, 0.1357446014881134, -0.5741636157035828, 1.380746603012085, 0.07072310149669647, -1.3523684740066528, 1.3921667337417603, -1.5813782215118408, 2.149583578109741, -0.2652876675128937, -1.4323270320892334, -0.11894994229078293, -0.16223494708538055, 0.9613552093505859, -1.0921272039413452, 0.10212782770395279, -0.20085890591144562, 0.23715025186538696, 2.3594882488250732, 0.12594521045684814, 0.951949954032898, 0.8904781937599182, -0.7467918395996094, 0.37373507022857666, -0.5307387709617615, 0.11373275518417358, 0.190577894449234, -1.355502724647522, -0.05733778700232506, -0.016451958566904068, -0.5365913510322571, -0.35414645075798035, -0.13763198256492615, 0.7812965512275696, -2.759047746658325, 0.9310671091079712, -0.8653281927108765, -0.45197710394859314, -0.10719667375087738, -0.32028812170028687, 1.6855539083480835, 1.7301833629608154, 1.4634593725204468, 0.6424247026443481, -0.3133474588394165, 0.18490062654018402, -0.22437581419944763, -0.14750589430332184, -1.7179380655288696, -1.0333269834518433, -1.9793195724487305, -0.9155100584030151, -1.3789390325546265, -1.6483147144317627, -0.7224404215812683, -0.1978534609079361, 0.8029690384864807, 0.43570035696029663, -0.5579230189323425, 0.48747822642326355, -0.3591136336326599, -1.3466600179672241, -0.3150006830692291, -1.4207478761672974, -0.06764760613441467, 0.34148380160331726, -0.6044144630432129, 0.9207479953765869, 0.199763223528862, 1.0221737623214722, -1.485398292541504, -0.8786998987197876, 0.13273252546787262, 0.9604668021202087, -0.7722238302230835, 2.517333745956421, 1.075416088104248, -0.49800944328308105, 0.3418436050415039, 0.13033223152160645, 1.474521517753601, 0.3862214684486389, 0.6171156764030457, -0.1432211548089981, -0.4436781704425812, 2.118885040283203, 0.051725395023822784, -1.2803373336791992, -2.276148796081543, 0.25572413206100464, 1.002676010131836, -0.772570013999939, 1.0906904935836792, 1.2823137044906616, -0.39408817887306213, 1.5125784873962402, -0.6387496590614319, 1.5025006532669067, 0.328442245721817, -0.2943398058414459, -0.22262564301490784, 1.879928708076477, 1.3065708875656128, 0.8097969889640808, 1.1020458936691284, -0.5832034349441528, 0.8217530250549316, -0.12854216992855072, -2.311415672302246, -0.5502342581748962, -0.46570533514022827, 1.0964157581329346, 0.9090166687965393, 0.7267894148826599, 1.1768250465393066, -0.96485435962677, 0.4757363498210907, 0.9256741404533386, 0.5189903974533081, 0.1474035382270813, 0.7507160902023315, 0.4723265767097473, -0.1585070639848709, -1.7700722217559814, -0.4580232501029968, 1.4440099000930786, -0.8192433714866638, 1.3438842296600342, -0.924401581287384, 0.34479984641075134, 1.6190922260284424, -0.6061925888061523, -2.4506218433380127, -2.108849048614502, 1.1475780010223389, -0.517723560333252, -0.013140833005309105, 0.014499245211482048, -2.0279226303100586, 0.9616302251815796, -0.3062874972820282, -1.0759483575820923, -0.6126201748847961, 0.5065945982933044, 1.1622259616851807, -1.3818166255950928, -0.1666765958070755, -0.35749924182891846, -0.6178144812583923, -0.753210723400116, 0.15101554989814758, -0.3201518654823303, -0.22941076755523682, 1.195673942565918, -0.6895388960838318, 0.6883900761604309, 0.9985085129737854, -0.5495626926422119, -1.6502810716629028, 0.6778640151023865, -0.04140119627118111, 2.6409542560577393, -1.7537258863449097, -0.20723193883895874, 0.18923713266849518, -0.5507726669311523, 0.48320508003234863, -0.063860222697258, 1.0314781665802002, -0.1155523806810379, 1.226424217224121, -1.6527736186981201, 1.2284395694732666, -1.284756064414978, -0.7716148495674133, -1.1112555265426636, 1.0765324831008911, -1.092193603515625, 2.2306883335113525, 1.4824215173721313, -1.3096723556518555, -0.19282691180706024, 1.2925902605056763, -0.5667517185211182, 0.8115159273147583, -0.665241003036499, 0.5301249027252197, 1.351232647895813, -0.6949983239173889, 0.5751010179519653, 1.500928521156311, 0.08806052058935165, 0.7926902174949646, 1.6922074556350708, -1.0123635530471802, -0.13380314409732819, -0.12366093695163727, -0.46845993399620056, 0.11201753467321396, 0.8219510912895203, 0.7073156833648682, 0.47253507375717163, 1.3475347757339478, -1.9419811964035034, 0.013839265331625938, 0.2601318359375, 1.1161338090896606, -0.6709909439086914, -2.405747413635254, -0.3764471411705017, 0.7490984201431274, -0.40178182721138, -0.7951902151107788, -0.22851668298244476, -2.099113941192627, -0.7114925980567932, -0.4047562777996063, -0.39484724402427673, -0.7158001661300659, -0.0672750324010849, -0.5738480687141418, 0.6310867071151733, 0.5265926718711853, 0.009684920310974121, 0.35619354248046875, 0.7110412120819092, -0.2371293008327484, 0.8653296232223511, -0.45388564467430115, 1.0306611061096191, 0.03447239473462105, -0.9789969325065613, 3.156407117843628, -0.2043670266866684, 0.4690248966217041, 0.39213576912879944, 1.1395267248153687, 0.16809381544589996, 1.2311668395996094, 0.2339927703142166, -0.6020845770835876, -0.266566663980484, 0.44144120812416077, 1.3655099868774414, 0.29608774185180664, -0.6139261722564697, -0.3136735260486603, -0.924456775188446, 1.3428735733032227, 0.09833524376153946, 1.7076499462127686, 1.9476889371871948, 0.8231231570243835, 2.4337825775146484, 0.051844194531440735, 0.09821991622447968, 0.9402470588684082, -0.7266261577606201, -0.36270928382873535, -0.530858039855957, 0.20396824181079865, -2.0631885528564453, 0.2121661901473999, -2.2111287117004395, 1.810041069984436, -0.985969603061676, 1.9236737489700317, 0.679604709148407, -1.7351250648498535, 0.6170893311500549, 0.05367188900709152, 1.0805457830429077, -2.680426836013794, 0.5083187818527222, -2.6686434745788574, -0.928369402885437, -0.11631263047456741, -1.0024011135101318, 0.029451478272676468, 1.5988795757293701, -0.15503904223442078, 1.021968126296997, 0.33463582396507263, 0.22290490567684174, 1.4597464799880981, -0.529633104801178, 0.7654114961624146, 0.04845127835869789, 1.4375312328338623, 0.6420692205429077, 0.3731142282485962, 0.7315384745597839, 0.8260882496833801, 0.3578556776046753, -0.839724063873291, 0.02023354172706604, 0.13766597211360931, -0.36232542991638184, -0.0431087501347065, -0.5331899523735046, -0.8391379714012146, -0.06918900460004807, -1.339247226715088, -0.3958874046802521, 0.158864364027977, 0.2074221670627594, 0.5040890574455261, -0.0375407449901104, -1.1440417766571045, 0.5733241438865662, 1.2525163888931274, -0.9849640130996704, 0.6343521475791931, 1.263741374015808, 1.0110979080200195, -0.48353707790374756, 0.07245168089866638, -0.24270130693912506, 0.3851463496685028, 0.308428019285202, -0.5689866542816162, -0.6406121253967285, 1.111171007156372, 0.8219889402389526, -1.3003073930740356, 0.4770187437534332, -1.0537109375, -0.3619145452976227, 0.8263770341873169, -0.35818278789520264, 0.8434844017028809, -0.2865544557571411, 2.186445474624634, 0.5392289161682129, 0.20186062157154083, 1.321319818496704, 0.5860487818717957, -1.0297945737838745, -0.29132354259490967, -0.508542537689209, 0.12341994047164917, -2.0996415615081787, 0.27487364411354065, 0.6421018838882446, 0.5565525889396667, -0.13157853484153748, -0.0036075422540307045, 0.025448257103562355, -0.7868512272834778, -1.1664400100708008, 1.5672537088394165, 0.5828660726547241, -2.5337460041046143, 1.8333848714828491, 0.7972592115402222, 1.081610083580017, -0.19955457746982574, 0.3956054449081421, -0.1508001834154129, 1.3605774641036987, -0.5633552670478821, 1.738113522529602, 1.127913475036621, 0.6737779974937439, -0.21344396471977234, -0.36410537362098694, 0.16691496968269348, -0.8484618067741394, -0.23089708387851715, -1.9676724672317505, -0.5686438679695129, 1.0088505744934082, -0.30951905250549316, -0.029784105718135834, 1.2848941087722778, -1.6277979612350464, 0.5683666467666626, 0.1461750566959381, -0.3729272782802582, 0.9899044036865234, -0.8093100786209106, -1.1172305345535278, -1.4169971942901611, -0.7745769023895264, 0.7901321053504944, -0.7189250588417053, 0.23753155767917633, -1.999110460281372, -0.509871780872345, -0.9794833660125732, 0.327947199344635, 1.5513640642166138, -1.5856738090515137, 0.7024980783462524, -0.24025548994541168, 0.6565641164779663, 0.9468686580657959, -0.5169663429260254, -0.48439180850982666, -1.1127800941467285, 0.9312704801559448, 0.18247279524803162, -0.9490982890129089, 1.877720832824707, 1.210726022720337, 1.054427146911621, -0.35538098216056824, 1.1013875007629395, 0.3187961280345917, 0.0036861218977719545, 0.5721504092216492, -0.7617554664611816, 0.8059507012367249, 0.9941821098327637, 0.7733273506164551, -0.9043209552764893, -1.8657827377319336, -0.453950434923172, -0.6969066262245178, 1.2065277099609375, -0.28337112069129944, -0.48555517196655273, -0.04894184693694115, 0.5365262031555176, -1.0081084966659546, -0.6060822010040283, -0.0746813639998436, 0.6552274823188782, 1.5985914468765259, -0.46440353989601135, 0.28408291935920715, -0.9255672097206116, 2.523019552230835, 0.6211013793945312, -0.3221706449985504, -1.0122921466827393, 0.5483455061912537, -1.2346720695495605, 0.549351692199707, 0.45695701241493225, 0.5579347014427185, 1.3589545488357544, 0.03452787920832634, 0.22797828912734985, -0.06432504206895828, 0.8727958798408508, 0.47058090567588806, 0.7253384590148926, 0.42856478691101074, -0.2316974252462387, 0.3665022850036621, 1.7838064432144165, 0.21013861894607544, -0.10556342452764511, 0.5270196199417114, -2.1256871223449707, -0.5651271343231201, 0.44898736476898193, 0.23133507370948792, 0.35584723949432373, 1.6335455179214478, 1.7648578882217407, -2.7454776763916016, 0.7540259957313538, 0.03333726525306702, -1.259587287902832, 0.9855910539627075, -0.2395571619272232, -0.1887785643339157, -0.32985323667526245, -0.0920727476477623, 0.619830310344696, -1.2290267944335938, 0.5635700225830078, 0.07446375489234924, -0.48835477232933044, 0.5754870176315308, -0.7017719745635986, -0.6786472201347351, 2.3219733238220215, 0.01772570051252842, -2.1581733226776123, -0.6657453179359436, -0.3771425783634186, 0.601593017578125, 0.9660354852676392, -1.8918238878250122, -0.8353382349014282, 0.5873802304267883, -2.449697256088257, 1.551697015762329, 0.4793042540550232, 0.5080601572990417, 0.32965925335884094, 1.6649478673934937, -0.0874190703034401, -0.8550466299057007, 0.010086690075695515, 0.16907669603824615, 1.4270590543746948, 0.04244695231318474, -0.283536434173584, 3.528998613357544, 0.14704489707946777, -0.45434844493865967, -1.342877984046936, 0.11520252376794815, 0.24800723791122437, 0.25021350383758545, -0.18530958890914917, -0.3236888647079468, -0.2747615873813629, -1.1832233667373657, 0.04611097276210785, 0.421459436416626, 2.144230604171753, 1.3285434246063232, 1.7367284297943115, -0.42235079407691956, -0.4235961139202118, 0.48027291893959045, -0.20971158146858215, 0.10529594868421555, 1.3207950592041016, 0.052518054842948914, 1.4524242877960205, -0.6519560217857361, -0.21359342336654663, 2.5889811515808105, -0.1799929141998291, -1.9449132680892944, 1.2904585599899292, 0.07027480751276016, -0.09911802411079407, 0.056209467351436615, -0.6921695470809937, 0.3333355188369751, 1.9885486364364624, 1.1484404802322388, -2.7294998168945312, 0.35625308752059937, -0.5934345126152039, -0.6444137692451477, -1.1507049798965454, -1.0041528940200806, 0.124348945915699, 0.17368236184120178, -0.6540916562080383, 0.47168201208114624, 0.4855809211730957, 0.20898433029651642, -0.0897359848022461, 1.478831171989441, 0.4475386142730713, -1.1917227506637573, 1.7736784219741821, -0.2615550756454468, -0.03831328824162483, -0.13272924721240997, 0.28958651423454285, 1.6000057458877563, -0.1122845858335495, 0.6291082501411438, -1.2381367683410645, -1.0747367143630981, 0.9762699604034424, -0.9796501398086548, 0.7115514278411865, 0.39353856444358826, 1.6191867589950562, 0.058424536138772964, -0.14195841550827026, -1.1505814790725708, -1.0054725408554077, -0.950911283493042, -0.625560462474823, 0.620026171207428, -0.13151785731315613, 0.3928034007549286, 0.9246670603752136, -0.6129281520843506, 2.46970272064209, -2.3870506286621094, -1.2135870456695557, 2.2320027351379395, -1.5892542600631714, -0.40561410784721375, 0.2576947510242462, -0.7724932432174683, 0.6123501658439636, -1.1765080690383911, 1.0044050216674805, -0.05304332077503204, -0.07158555835485458, -0.1302061378955841, 1.6043106317520142, -0.5189940929412842, 0.25112631916999817, -1.2422555685043335, -0.7265634536743164, 1.313132405281067, 0.7808522582054138, -0.953655481338501, 0.7284432053565979, -1.1941041946411133, 0.6774944067001343, 1.7383004426956177, -1.3311406373977661, -2.2026116847991943, -0.6952189803123474, 0.705811083316803, -1.7318708896636963, 1.3386718034744263, -1.0669329166412354, -0.6294852495193481, -0.2371278554201126, -0.6864777207374573, 0.9868190288543701, 0.3755718171596527, -0.3665721118450165, -0.23874817788600922, 0.4469049870967865, 0.7840625047683716, 0.24341388046741486, 0.5300800800323486, 1.0113810300827026, -0.15264132618904114, -0.4868379235267639, 1.4920827150344849, -0.9115229249000549, 0.7536529898643494, -0.3977257311344147, 0.079417884349823, -2.4088685512542725, 0.08042476326227188, 0.8674007654190063, 1.6813077926635742, -0.3010284900665283, -1.2292598485946655, 0.9991876482963562, -0.9508534669876099, -0.9027997851371765, -1.138998031616211, -1.7135772705078125, 1.3452407121658325, -0.39809945225715637, 0.3321475088596344, 0.5205857157707214, -2.498396158218384, -1.4677729606628418, 1.0968115329742432, -0.5788096785545349, -1.4748932123184204, -0.89885014295578, -0.9736558794975281, 0.24031080305576324, -0.5444967746734619, -0.850774884223938, -2.1251723766326904, 1.2670788764953613, -0.906253457069397, 1.7019237279891968, -2.197040557861328], [-0.10923678427934647, -1.9928947687149048, -0.9842093586921692, -0.43862438201904297, 1.7654510736465454, -0.21581971645355225, 0.4136514365673065, -0.0824989452958107, -0.3840741813182831, 0.2572290301322937, -0.5873218774795532, -0.08020460605621338, 1.7618166208267212, 0.2687254548072815, 0.23013482987880707, 0.5845694541931152, -0.9359191060066223, -0.6043848991394043, -1.6414211988449097, -0.6344709396362305, 0.9137235283851624, 0.03317169100046158, -0.5548695921897888, -0.8896936178207397, -0.19362393021583557, -0.10502699762582779, -0.5523051619529724, 1.5538303852081299, -0.3103715479373932, -0.1374305635690689, -0.3345341682434082, -0.3963439166545868, -0.7265007495880127, -1.1732776165008545, -0.27191442251205444, -1.4564846754074097, -0.5314620137214661, -0.4675716459751129, -0.9695906043052673, -0.7354326248168945, 0.0390438511967659, 2.1411092281341553, 0.5775254368782043, 0.8705351948738098, -0.4798988401889801, 1.1129329204559326, -0.9179434180259705, -0.043903909623622894, 0.15280590951442719, -0.25554654002189636, -2.3558242321014404, 0.060159314423799515, -0.818478524684906, 0.8349703550338745, -0.8602612614631653, 0.46025949716567993, 0.1470213234424591, -0.8810440897941589, 0.964863657951355, -0.711185872554779, 0.9731341004371643, 0.8588390946388245, 0.4177412688732147, 0.482028067111969, -1.3237463235855103, -0.4550083875656128, -0.2634071111679077, -0.9982871413230896, 0.6652785539627075, 0.011277933605015278, -0.1508760303258896, -0.16906075179576874, 0.6729453206062317, 1.8600398302078247, -0.5904178023338318, 0.27497297525405884, 0.39288330078125, -0.012108023278415203, 0.03218008577823639, 1.3842531442642212, -0.8563089370727539, -0.3246903419494629, -1.147010087966919, 1.1777485609054565, 1.844517707824707, 0.06807883083820343, 0.5177220106124878, 1.03620183467865, 0.3730606436729431, -0.7727121710777283, -0.031178900972008705, -0.13056376576423645, 0.21690912544727325, 1.5544981956481934, 0.23518067598342896, 2.5009937286376953, 2.4885473251342773, -0.9688317775726318, -1.554734706878662, 0.6453157067298889, -0.8694596886634827, -1.0039986371994019, 1.5809203386306763, -1.0813161134719849, -0.5434192419052124, -0.9504050016403198, -0.76528000831604, 0.22371578216552734, -1.191132664680481, 0.48462459444999695, -0.7602728009223938, 0.26130762696266174, -1.5908904075622559, 0.628897488117218, 0.005586175248026848, 0.7465636730194092, 0.03317849710583687, 0.23405061662197113, 1.8258355855941772, 0.30184903740882874, -0.35820698738098145, 0.7099005579948425, -0.4105396866798401, 1.9888191223144531, -0.20934033393859863, 0.4531353712081909, 0.734798014163971, -0.43851834535598755, 1.262799859046936, 0.3392796218395233, -0.5890031456947327, 0.5196110606193542, 1.1368471384048462, 0.7210932970046997, 0.74669349193573, -0.6005375981330872, -0.3321274518966675, 0.6635586023330688, 0.5875251889228821, 1.1478919982910156, 1.0209559202194214, -0.05360208451747894, -0.29111185669898987, 1.0239921808242798, -0.6251680850982666, -1.5677708387374878, 0.30183014273643494, 1.457481026649475, 0.8696750402450562, 0.2510724365711212, -0.5201178193092346, 1.4471900463104248, 0.943487286567688, -2.015864133834839, 0.890275776386261, -0.8436047434806824, -2.303100824356079, -1.382900595664978, -2.125401735305786, -0.04270176216959953, 0.20300531387329102, -0.9677372574806213, 0.6952325701713562, -1.4137110710144043, -0.026462728157639503, 2.9403510093688965, 1.0664259195327759, 0.7329516410827637, 0.3073427379131317, 0.38521140813827515, -1.0353319644927979, -0.40980949997901917, -1.2697378396987915, -1.851631999015808, 0.004826812539249659, -0.02032260037958622, -1.0109466314315796, 0.1992301195859909, -1.607042670249939, 1.506408929824829, 0.02026684768497944, 0.6632246375083923, -1.110831379890442, -1.0416135787963867, -0.028762463480234146, 0.28088125586509705, -0.6596361994743347, 1.1648011207580566, 0.09702400863170624, -0.9767270088195801, -2.232097864151001, 0.4261474907398224, -0.9045896530151367, 0.16003923118114471, 0.825474739074707, -0.2776777744293213, 0.48218560218811035, -1.1716994047164917, -0.38843026757240295, 0.07607686519622803, 0.22647637128829956, 0.5147669315338135, -0.421235054731369, -0.06661589443683624, 0.5277059078216553, 1.3522770404815674, -2.062269926071167, -1.781765103340149, -0.45492023229599, 0.7861289978027344, 2.3264381885528564, -0.334005206823349, -0.028799360617995262, -1.0621821880340576, 0.5070537328720093, -0.857237696647644, -0.04933721572160721, -0.6079002022743225, 0.5411182045936584, -0.16269902884960175, -0.40845951437950134, 0.09232388436794281, 0.1445501744747162, 1.150396704673767, -1.0751259326934814, -0.4519203007221222, -1.3556770086288452, -2.2844607830047607, -0.0759207233786583, 1.8131595849990845, -0.5963416695594788, 1.0541393756866455, -0.02089918963611126, -0.2427612841129303, 1.1309999227523804, -0.3690631687641144, -1.178220510482788, -1.9974448680877686, -0.4397793114185333, -0.22162464261054993, 0.9342300891876221, -1.902975082397461, 0.8565585613250732, -0.15444256365299225, -1.223387598991394, -0.38741278648376465, -1.7142956256866455, 1.1979079246520996, -0.740484356880188, -0.8132268786430359, -0.4419446289539337, -0.6223046183586121, -0.45153313875198364, 0.5109038949012756, -0.4630489945411682, -0.04939043149352074, -2.0536792278289795, -0.43030819296836853, -0.171774223446846, 0.7722205519676208, -2.4080967903137207, 0.3433659076690674, 1.357240915298462, -0.12316610664129257, 0.2504599392414093, 1.5077574253082275, -0.1525021344423294, -0.6398797631263733, -0.7145118117332458, 0.6090172529220581, 1.242067575454712, 1.4698245525360107, -1.39653480052948, -0.29473453760147095, -0.9912469983100891, -0.7630349397659302, -0.7044434547424316, 0.7043373584747314, -2.347803831100464, -0.8501661419868469, -0.39144396781921387, 0.6474534273147583, -1.297204613685608, -0.24356266856193542, -0.6565669178962708, 0.964608371257782, -0.28585726022720337, -0.42575061321258545, -0.8289583325386047, -0.47391027212142944, -0.4148004949092865, -2.7128794193267822, -1.142521858215332, 0.32506993412971497, -0.8570404052734375, -0.6322765946388245, 0.2536426782608032, -0.7270416617393494, 1.7590916156768799, 0.6530952453613281, 0.22848692536354065, -0.6368524432182312, -0.8390272855758667, -0.03532678261399269, -0.9764424562454224, 0.7544510960578918, 1.1171953678131104, -0.9678980112075806, -0.5348863005638123, -0.9179381728172302, 0.4410566985607147, -3.014172315597534, -0.7020192742347717, -0.08922064304351807, 1.0315450429916382, -0.06219709664583206, 0.8451370000839233, 1.3664520978927612, 0.9469882249832153, 0.615069568157196, -0.1673559546470642, 0.05634072795510292, 0.6714391708374023, 0.25449052453041077, -0.5220453143119812, -0.9894566535949707, 0.9131186008453369, -0.32941192388534546, 0.1614585518836975, 1.5336799621582031, -0.5971282720565796, 0.5709930062294006, -0.153475821018219, 0.03192102536559105, 3.024970531463623, -1.052079677581787, -1.8733576536178589, -1.9799349308013916, -0.6747570633888245, 1.307981252670288, -1.4407237768173218, -1.4370172023773193, 0.3854818046092987, -0.7899080514907837, -1.3120794296264648, -0.5298646092414856, -0.6790552139282227, -0.6191251277923584, 0.30307722091674805, 1.0170416831970215, 0.8456810116767883, -1.2580726146697998, -0.15377627313137054, 0.8245067596435547, 0.12663309276103973, -0.11176557093858719, -0.617181122303009, -1.6880903244018555, 0.9243959188461304, 1.4629137516021729, -0.1901971995830536, 0.9984350800514221, 0.944168746471405, 0.36183303594589233, 0.7978253364562988, 0.48374587297439575, 1.7360979318618774, 0.6412339210510254, -0.42086872458457947, -0.1329607218503952, 0.02847268618643284, 0.13453060388565063, -0.09081199765205383, 1.181841492652893, 1.2181493043899536, 0.5228748321533203, -0.6049491763114929, -0.6585783958435059, 1.5429985523223877, -0.15221214294433594, 0.5212424993515015, 0.8628695607185364, -0.7404131293296814, 2.4388513565063477, -1.694197416305542, -0.26150310039520264, 1.0406972169876099, -0.6221868991851807, 1.1728863716125488, 1.5629395246505737, -1.429015874862671, -1.6881532669067383, 0.9794308543205261, -0.7338394522666931, -0.8978360295295715, -0.6976596713066101, 0.3220808506011963, 1.0201250314712524, -0.579777181148529, -0.134893536567688, 0.580224871635437, -1.258792757987976, 0.09928900748491287, 1.1219841241836548, -0.14984168112277985, -2.157740592956543, -1.0877883434295654, 0.33266621828079224, -0.07535523921251297, 1.093693733215332, 0.11382981389760971, -1.2915277481079102, -0.10395146161317825, -1.4869678020477295, -0.9298483729362488, -1.0398101806640625, 0.9938815832138062, -0.5754932761192322, -0.3339867889881134, -0.4430501461029053, -0.8191184401512146, -0.6374250650405884, -0.8480708003044128, 0.4342295825481415, -0.5051283240318298, 0.48753201961517334, 1.7112550735473633, -1.2675285339355469, 0.6985417604446411, -1.1028501987457275, 0.48252061009407043, 0.47302064299583435, 0.721534252166748, -1.391039490699768, -0.147711843252182, -1.3488000631332397, -1.1220296621322632, -0.741988480091095, -0.07978495955467224, 0.03809358552098274, -1.0787370204925537, 0.21413639187812805, -1.0239883661270142, -1.6742491722106934, 0.20572073757648468, -0.06038565933704376, -0.38503792881965637, 0.29362380504608154, -0.6165797114372253, 0.6010797023773193, 1.1191445589065552, -1.3313097953796387, 2.0238919258117676, -0.2592688500881195, -0.14236152172088623, -0.8697611689567566, 0.31381285190582275, -0.04509463533759117, 0.5850822329521179, 0.22365424036979675, 0.5658941864967346, 0.28739818930625916, -0.7924425005912781, 0.981886625289917, -0.9997291564941406, 1.3571449518203735, -2.2170746326446533, 0.4326150119304657, 1.6023211479187012, 0.015993423759937286, 1.5468080043792725, -0.737493634223938, 1.5455374717712402, -0.18504174053668976, 1.0161908864974976, 0.26314857602119446, -0.5813772082328796, 0.42543166875839233, -0.7011309862136841, -0.11874348670244217, -0.9960907697677612, -1.442328691482544, -1.525145173072815, 0.2160637080669403, -0.6538524031639099, 0.7397490739822388, -0.43277421593666077, 0.3875318765640259, 0.17840521037578583, 0.056098099797964096, -0.3959258794784546, -1.8461432456970215, 0.9882442951202393, -0.47279092669487, 0.927978515625, -0.38652855157852173, 0.3805455267429352, 2.3566269874572754, 0.35680246353149414, -0.28689146041870117, 1.3038709163665771, -0.24356438219547272, 0.4905979633331299, 1.4067742824554443, 0.30058029294013977, 0.014862076379358768, 0.4559093415737152, -1.3611743450164795, 0.6660845875740051, 0.13488651812076569, -0.4129836857318878, 0.940921425819397, 0.021147914230823517, 1.0076078176498413, -0.24848003685474396, -1.9584449529647827, -1.4393420219421387, -0.6615030169487, -0.3555871248245239, -1.2758151292800903, 0.9716453552246094, 0.5942462086677551, -1.1221389770507812, -0.3976673185825348, -0.08201132714748383, -1.1368359327316284, 2.203587055206299, 1.7558437585830688, 1.1652626991271973, -0.16112546622753143, -0.9185764193534851, 0.5088751316070557, -0.15385231375694275, -0.1213730052113533, -1.1206706762313843, -1.1853443384170532, -2.5160441398620605, 0.40388333797454834, 2.2044150829315186, -0.9438546895980835, 0.6808643341064453, 0.41386616230010986, -0.12863034009933472, -1.3087657690048218, -1.3912333250045776, -0.2894139289855957, 1.0743789672851562, -1.0538361072540283, 0.08275479823350906, 0.2526083290576935, 1.1666697263717651, -0.1586730182170868, -0.6079248785972595, -0.9092832207679749, -0.3908924460411072, 0.04725928232073784, -0.23409131169319153, -0.9589475989341736, 0.6177859306335449, -1.245836853981018, 2.340229034423828, 0.1464415341615677, 0.6951163411140442, 1.130428433418274, -0.647787868976593, 1.1080031394958496, -0.22949472069740295, 0.45764219760894775, 2.1231138706207275, 0.5131584405899048, -0.28642749786376953, -1.7835720777511597, 1.1973932981491089, 1.4311944246292114, 0.12791022658348083, -0.38501545786857605, -0.004544428549706936, 0.2465556412935257, 0.9450863003730774, 0.45221567153930664, -1.8282023668289185, -0.3810034990310669, -0.5859844088554382, 0.8233801126480103, 0.2863021790981293, 0.3047959506511688, -0.4973081350326538, -0.017800334841012955, -0.9220050573348999, -0.25650280714035034, -0.06694316864013672, 1.0931973457336426, -0.34811869263648987, 0.06187688186764717, 1.1448063850402832, -0.489458292722702, 0.2837347686290741, 0.7805826663970947, 0.2221803367137909, 1.7809240818023682, 0.4408312439918518, -0.2948618531227112, -0.5427103638648987, -1.5502663850784302, 0.9408434629440308, 1.415736436843872, 0.08243909478187561, -2.3157799243927, -0.18441516160964966, 0.8780778050422668, -0.8231678009033203, 1.4728323221206665, -1.91706120967865, -0.831325888633728, -0.3233299255371094, 1.088972806930542, -1.3969624042510986, -1.5352494716644287, -0.12216080725193024, -0.4843716323375702, -1.2338734865188599, -1.3613616228103638, -0.36922556161880493, 0.5537199378013611, 0.0881023183465004, -0.27199214696884155, -0.11166507750749588, -0.6603319048881531, -0.607254147529602, 1.9436389207839966, -0.21718484163284302, -0.007114951033145189, 1.1006568670272827, 0.14555233716964722, 1.1355096101760864, -0.5063399076461792, -0.5034055113792419, -1.6476778984069824, 0.5235735177993774, -1.3979239463806152, 0.9097461700439453, 0.6988195180892944, -0.6613358855247498, 0.2965221107006073, -1.5377825498580933, -1.1367721557617188, -0.15673856437206268, -1.1837387084960938, -0.14519251883029938, 0.24446777999401093, 0.2758399546146393, 0.04942524805665016, -0.3730582594871521, 0.3644373416900635, 1.6095303297042847, 0.20095056295394897, -2.2225847244262695, -0.40037861466407776, -1.4851694107055664, -1.2832733392715454, -1.183621883392334, 0.7138612270355225, -1.5918500423431396, -0.8372606039047241, -1.0824013948440552, 0.7951995730400085, -0.02183798886835575, 0.11199931800365448, -0.5552125573158264, -0.9599426984786987, -1.3046022653579712, 0.7139918208122253, 1.101257562637329, 0.6722928285598755, -0.5659471750259399, 1.4850096702575684, -1.375382661819458, -0.16255882382392883, 1.0466793775558472, 0.18331532180309296, -0.2899234890937805, -0.23189204931259155, -0.6548209190368652, -0.24492929875850677, -0.4717070460319519, -0.6582992672920227, -1.0774462223052979, 0.34142857789993286, -1.1019953489303589, 0.8429707884788513, 1.1143826246261597, -1.1165157556533813, 1.7896851301193237, 0.42710551619529724, 0.3756309747695923, -1.3217823505401611, 0.9090893864631653, -1.3872309923171997, -0.4106830060482025, -1.9753427505493164, -0.32906869053840637, -0.6481149196624756, -0.8565412759780884, 0.734565794467926, -0.849860668182373, 0.3142099380493164, 1.698317527770996, -2.021967649459839, -0.022893140092492104, 2.029155969619751, 0.4706895351409912, -1.528842806816101, -1.3997604846954346, 0.16451843082904816, 1.7270221710205078, -0.18884530663490295, 0.143876850605011, 0.407675564289093, 0.7530702352523804, 0.8583859205245972, -0.025733649730682373, 1.8080391883850098, -1.6957824230194092, -1.0071485042572021, -0.25804373621940613, -0.24631519615650177, 0.07376208901405334, -0.25311413407325745, -0.681720495223999, 0.6181125640869141, 0.3735497295856476, -0.6820499897003174, 0.933673620223999, 0.6946303844451904, 1.1874682903289795, -0.7904881834983826, -0.1453528255224228, 0.7533801198005676, -1.2179055213928223, -0.3312230706214905, 1.257176160812378, -0.13817250728607178, -1.5245779752731323, 1.32346510887146, 0.43037036061286926, -1.424286127090454, -1.0603288412094116, 0.5762232542037964, -1.252333164215088, 1.7859746217727661, -0.10188886523246765, -0.6343396902084351, 0.05197601020336151, -1.710954189300537, -0.18239544332027435, -0.6730378866195679, -0.6756373643875122, -0.032282475382089615, -1.2309406995773315, 1.460654854774475, 0.30981624126434326, 1.12625253200531, 0.440564900636673, 1.180478811264038, 0.3756849467754364, -1.1106394529342651, 1.562617540359497, -0.7571555376052856, 0.8937734961509705, 1.4354411363601685, 2.2364108562469482, 1.5026224851608276, 0.8934210538864136, 0.5572845339775085, 0.6846016049385071, -0.09265875816345215, 0.6753100752830505, -1.3640005588531494, 1.3243881464004517, 0.062204375863075256, 0.3964911699295044, -0.18598513305187225, -0.47146937251091003, 0.011140808463096619, -2.0915894508361816, -0.3117409646511078, -0.004029921256005764, 0.9387074708938599, -0.3856276571750641, 0.8878220319747925, -0.11403834074735641, -0.046460021287202835, -1.1595364809036255, -1.1276355981826782, -0.30162009596824646, -0.2358129620552063, -0.2318439781665802, 0.46178004145622253, 0.03636140376329422, -1.5960537195205688, -1.6269588470458984, 0.19059047102928162, 0.20188091695308685, 0.50902259349823, 1.1920126676559448, 0.02345449849963188, -0.6831636428833008, 0.4027053713798523, -0.3865188658237457, 1.6362460851669312, 1.7003133296966553, -1.3272430896759033, -0.6854015588760376, 1.2922875881195068, -0.4045194983482361, -2.3106768131256104, -1.201305627822876, -1.3598852157592773, 0.5357407331466675, -0.38538631796836853, 0.615230143070221, 1.3513259887695312, 0.28299108147621155, -0.001158506260253489, 0.3306460678577423, 0.5678387880325317, 0.531223714351654, -1.7837904691696167, 0.573945164680481, -0.027037933468818665, 0.4189022481441498, -0.7691303491592407, 1.1477065086364746, 0.6737046241760254, 0.22526736557483673, 0.5405304431915283, -0.626831591129303, 0.22536903619766235, 0.4685947299003601, 0.49650993943214417, -1.1129547357559204, -1.0384339094161987, -0.32837221026420593, 0.05162481963634491, -1.9409462213516235, -0.9244979023933411, -0.04848269745707512, 0.2614968717098236, 1.3650100231170654, 0.739894449710846, 0.9311519861221313, 1.0112706422805786, 0.19240906834602356, 0.043288931250572205, -0.5761092305183411, 0.8068647384643555, -0.6166872978210449, 0.5642039179801941, -0.03927149251103401, 1.4864521026611328, -0.5985969305038452, 0.47488248348236084, -0.11800936609506607, 0.5486844182014465, 1.3072658777236938, -0.8011056184768677, 1.220697045326233, -1.481366515159607, 0.1506991684436798, -0.8253618478775024, 0.10580774396657944, -0.5989008545875549, -0.8792195916175842, -0.614417552947998, 0.041009850800037384, -0.34431153535842896, 0.6869325041770935, -0.045393917709589005, -1.0836117267608643, 0.5453572273254395, -1.6733564138412476, -0.4531749486923218, -1.8213955163955688, 1.3460350036621094, 1.304032564163208, 0.5875391960144043, 0.23981158435344696, 2.539898633956909, 0.7247121334075928, -0.36467617750167847, 1.1775963306427002, 2.2537426948547363, -0.8090255856513977, 1.215440273284912, -0.3924964964389801, -1.7917506694793701, -1.8060007095336914, 0.5170842409133911, 1.3273155689239502, 2.1129989624023438, 1.0834354162216187, -1.3987830877304077, -1.82707679271698, -0.49050354957580566, -1.0938546657562256, 2.039374828338623, 2.6563198566436768, -1.3696417808532715, 0.6991721391677856, -0.06175683066248894, -0.235894575715065, 1.193581223487854, -0.11030649393796921, -0.5402369499206543, 2.578223943710327, 0.728315532207489, -0.6310288310050964, -0.13434667885303497, 0.2367815375328064, 0.5368508696556091, 1.2722837924957275, 0.2910730242729187, -0.33216360211372375, -0.4190911650657654, -0.8028085827827454, -1.3358774185180664, -0.6150727868080139, -0.8975619673728943, -1.1291130781173706, 0.45172661542892456, 3.1032698154449463, 0.09856581687927246, -0.8463859558105469, -0.881201446056366, 0.45039427280426025, 0.10004077106714249, -0.7245069742202759, -0.3264690041542053, -0.6130755543708801, -1.1359522342681885, 0.6418103575706482, 0.19245770573616028, -0.400383323431015, -0.7920053005218506, 0.5655949711799622, 1.6767529249191284, 1.416383981704712, 1.6655199527740479, 0.6786177754402161, 0.02988579496741295, 0.8863828778266907, -0.5679263472557068, 1.0263185501098633, -0.010060802102088928, 0.7274292707443237, 0.7148516774177551, 0.07646748423576355, -0.21554149687290192, 0.8998170495033264, 0.771843433380127, 0.06097419187426567, -1.8018715381622314, 0.426641047000885, -0.32418060302734375, -1.8552179336547852, -0.09174534678459167, 0.17697712779045105, -1.217574119567871, -0.6480319499969482, -1.5412561893463135, -1.1632314920425415, 1.9246466159820557, 0.4524883031845093, 1.4058626890182495, -0.7837639451026917, 1.5811058282852173, 0.3430996835231781, -0.20007191598415375, 0.21315675973892212, -0.907489001750946, -1.4120253324508667, -0.6792312264442444, 1.0282725095748901, 0.5030109882354736, 0.13964033126831055, -0.06201533228158951, -0.7413643002510071, 0.25721874833106995, 0.44079288840293884, -0.16950993239879608, -0.596048891544342, -1.3016120195388794, -1.2754943370819092, 0.40518316626548767, 0.16426365077495575, 0.1990589052438736, -0.0571611151099205, 0.4129643440246582, -0.2905842661857605, 0.4086572825908661, -0.9546941518783569, -0.5150209069252014, 1.519044280052185, 1.6052820682525635, -0.37585118412971497, 0.5920390486717224, 0.6081930994987488, -0.7433005571365356, -0.24475912749767303, 0.9191933870315552, -0.5140918493270874, -0.20581190288066864, -1.176160454750061, -0.9821814298629761, 0.4699499309062958, 0.9190665483474731, -1.0113255977630615, 0.3010188043117523, -1.4496502876281738, 0.34206435084342957, 0.8363211750984192, 0.35730043053627014, -1.5175443887710571, -2.0264954566955566], [0.8990730047225952, 0.7825565934181213, -1.0837539434432983, 0.34586963057518005, 0.4172404408454895, 0.7950741648674011, 0.8386818766593933, 0.9231283068656921, 0.7695766687393188, 0.3325510025024414, 0.35893553495407104, 0.23130732774734497, -0.7176324725151062, 0.976266622543335, 1.5350977182388306, 0.4556376338005066, 0.26387712359428406, 0.033342402428388596, -1.187419056892395, -0.6769181489944458, 1.9907482862472534, -0.9307963252067566, 1.2515287399291992, 2.111158847808838, 0.0027621504850685596, 1.876061201095581, 0.15656778216362, -0.6947239637374878, -2.7661900520324707, -0.23518553376197815, 0.67044997215271, 1.7614843845367432, 1.0964888334274292, -1.0629539489746094, -1.2254703044891357, -0.21104003489017487, -0.505435049533844, 0.2671331763267517, -0.24496039748191833, 0.023957423865795135, -0.5089598298072815, 0.8210207223892212, 1.9305896759033203, 2.098369598388672, -2.184556007385254, 0.04194559156894684, -1.3139946460723877, 0.4314973056316376, -0.008074753917753696, -0.4346219599246979, 0.9048510193824768, 1.3223876953125, 0.09051782637834549, 1.3740590810775757, 0.7128576636314392, -0.3475452661514282, 0.3709988296031952, -1.1594204902648926, 0.01522227842360735, -0.5331947207450867, -0.07521861046552658, -2.259742021560669, 0.8769294619560242, 0.05044452100992203, 0.7060330510139465, -0.5628748536109924, -1.1388441324234009, 1.3951512575149536, 0.8341684341430664, -0.8128337860107422, 0.13154791295528412, 0.6555201411247253, -0.7023847699165344, -0.361355721950531, -0.42225542664527893, -0.7810598611831665, 0.11982575058937073, 0.7184675335884094, -1.4343317747116089, -0.06986214965581894, -0.44774624705314636, -0.0290497038513422, 0.7356098890304565, 1.3187556266784668, -0.7049374580383301, -1.0684114694595337, 0.6195653676986694, -0.8107284903526306, 1.105195164680481, 1.3994572162628174, 0.3949160873889923, 0.7001571655273438, 0.42177438735961914, -1.7879184484481812, 3.0511348247528076, -0.41059958934783936, -0.5346603989601135, -1.5939584970474243, 0.3612322509288788, 1.0716673135757446, -0.8790125846862793, -0.4193083345890045, 0.8947025537490845, 0.020773185417056084, 0.4573787450790405, 0.6757091879844666, -0.016105538234114647, 0.003869028063490987, 0.5437156558036804, -0.43679314851760864, 0.47140073776245117, -1.001146912574768, 1.2428841590881348, -0.2122362107038498, 1.6017993688583374, -0.17514705657958984, -2.0408430099487305, 1.0232418775558472, 1.11996328830719, -0.031222837045788765, -0.9225738048553467, -1.246753215789795, -0.1573319286108017, 1.0410585403442383, -0.9246777892112732, 0.08252132683992386, -1.3359131813049316, 1.0121136903762817, -0.6986452341079712, 0.727344274520874, -2.3511228561401367, 0.8893091082572937, -0.3068373501300812, 0.36881980299949646, -1.0449954271316528, 1.2129312753677368, -0.19497238099575043, 2.421834945678711, 1.7307579517364502, 0.6281998157501221, -1.5610440969467163, -0.2919023633003235, -0.5643273591995239, 0.3020873963832855, 1.5101081132888794, 0.18731720745563507, 1.184328556060791, 0.02080666273832321, 0.7442278265953064, 0.30299949645996094, 0.36975064873695374, -2.4483304023742676, -0.4799951910972595, -1.0401790142059326, -0.08690708130598068, 0.38213908672332764, 0.15551123023033142, 0.07742185145616531, -0.4690113365650177, 0.5335606336593628, 0.4106910526752472, -1.1948957443237305, -0.08008424937725067, 0.24630104005336761, -0.20442144572734833, -1.2334493398666382, 0.3614746034145355, -1.079331398010254, 0.6547226309776306, -1.0386914014816284, 1.4149681329727173, 0.6733148694038391, 0.46836039423942566, 0.7277114391326904, -0.05162801221013069, 0.8938502073287964, -0.6006649732589722, -0.18558865785598755, -1.2032426595687866, 0.5017094016075134, -0.44754746556282043, 0.10560037195682526, -0.13784343004226685, 1.4032231569290161, 1.4091687202453613, 0.5299085974693298, 1.9369912147521973, 0.3392760753631592, -0.753204882144928, 1.4098420143127441, 1.173693060874939, 0.2705393433570862, 0.7281007170677185, 3.9767863750457764, 0.8873664736747742, -1.3645853996276855, -1.4871138334274292, -0.5918322801589966, -2.0912349224090576, 0.5577258467674255, -0.026062527671456337, -2.6412854194641113, -0.6947817802429199, 0.034277718514204025, 1.1163041591644287, -0.5137410759925842, 0.659095287322998, 1.0887020826339722, -1.1558350324630737, 1.0994720458984375, 1.0747873783111572, -2.2856521606445312, 0.7662283778190613, -0.7023481726646423, -0.6826210021972656, 0.7075629234313965, -0.5454716682434082, 1.253952145576477, -0.7273195385932922, 0.34894031286239624, 0.1072380319237709, -0.6348218321800232, -0.37848326563835144, -0.21157652139663696, 0.6628561615943909, 1.9527949094772339, 1.964545488357544, 0.2051505744457245, -0.12012828886508942, -1.4948666095733643, -1.436686635017395, 1.408774495124817, 0.2066403031349182, -1.8794864416122437, 0.45904117822647095, 0.9039685130119324, -0.9393182992935181, -0.7402095794677734, -1.7803200483322144, 0.26712536811828613, 0.1478302925825119, 1.0516149997711182, 1.805850625038147, -0.6651919484138489, -0.2759137451648712, -0.21721826493740082, 1.7611851692199707, -0.32730138301849365, 1.5403110980987549, -1.2225152254104614, 0.5526601076126099, 1.288831353187561, -2.1756110191345215, 0.4987325668334961, 0.4374973475933075, 0.48947399854660034, -0.1697925180196762, 0.7696170210838318, -0.7769981026649475, 0.41568630933761597, 0.49925535917282104, -0.616334855556488, -1.1137280464172363, 0.5171447396278381, 0.30931445956230164, -0.32031792402267456, -0.4977595806121826, 1.3097394704818726, 1.5873425006866455, -1.495802879333496, -0.6477293372154236, 0.5277361869812012, -0.7682585716247559, -0.8714060187339783, 0.13999435305595398, 0.1999172568321228, -0.09962671995162964, -1.7835654020309448, -1.293740153312683, -0.638173520565033, -0.3662249445915222, -0.3603212237358093, -0.3928772211074829, 1.1617556810379028, 0.09792570769786835, 0.40411505103111267, -0.5036342144012451, -2.630674123764038, 1.1383717060089111, 0.4972988963127136, 0.5995481610298157, 0.10089703649282455, -1.319339632987976, -0.23257066309452057, 0.03761066868901253, -0.49116820096969604, 0.4761325716972351, -0.9104302525520325, -0.34701740741729736, -0.18279463052749634, 0.29731160402297974, -0.49798476696014404, -0.36498305201530457, 1.2858331203460693, -1.3794209957122803, 2.2367095947265625, -0.25648775696754456, 0.37340041995048523, 0.3004186749458313, 0.9119196534156799, 0.6696984767913818, 0.12132006138563156, -0.45717310905456543, 1.1361595392227173, 0.4425353407859802, -0.1061289831995964, -0.6755474209785461, -0.020029598847031593, 0.13951045274734497, -1.0217821598052979, -0.5941863656044006, -0.11840187758207321, -0.8148961663246155, 0.3458222448825836, -1.0657652616500854, -2.577728509902954, -1.3562278747558594, 0.5511537194252014, -0.15607500076293945, 0.18151529133319855, -0.31556668877601624, 0.5110117197036743, -0.06655069440603256, 1.5038259029388428, -0.3556280732154846, 0.32978010177612305, 1.0835437774658203, 2.4494245052337646, -1.6553136110305786, -0.16000133752822876, 1.1986424922943115, -0.15540711581707, -1.837795376777649, 0.8168927431106567, 0.0758134052157402, -1.6237554550170898, 0.5164626240730286, 0.19098788499832153, -1.2888789176940918, 0.2870204448699951, -2.269050359725952, 1.3164150714874268, -1.9551678895950317, -0.45102420449256897, -0.9252135753631592, 1.6994967460632324, 0.8548727035522461, -0.3779711127281189, 1.493988275527954, -0.9236598610877991, 0.09540138393640518, 0.43659478425979614, -2.1536786556243896, 0.8474693894386292, -0.13360053300857544, 0.1530400812625885, 1.40376877784729, 0.6253318786621094, 0.8992210626602173, -0.18071506917476654, -0.1610267460346222, 1.9017866849899292, 0.34850218892097473, -0.1126171201467514, -0.5895978212356567, 0.301757276058197, 1.3691813945770264, 1.2357350587844849, 0.7500403523445129, 1.3881549835205078, 0.4999644458293915, -0.28177884221076965, 0.41676563024520874, -1.7219092845916748, 0.3983711302280426, 1.198500394821167, -0.8352622389793396, 1.2864083051681519, -0.8822145462036133, -1.492537021636963, -1.150799036026001, 0.10863932222127914, -1.2763172388076782, -0.8875249624252319, 1.031685471534729, -0.42088913917541504, -1.0157926082611084, 1.813493251800537, 0.15091396868228912, 1.7524999380111694, 1.2285150289535522, -0.10008573532104492, 1.1141141653060913, 1.3133922815322876, 0.12792769074440002, -1.1454498767852783, 0.20182432234287262, 0.26588189601898193, -0.9762756824493408, 1.094271183013916, 0.7603047490119934, 0.7253145575523376, 0.997827410697937, -1.0469975471496582, -0.9234312176704407, -1.03429114818573, 1.048727035522461, -1.0246012210845947, 0.37618404626846313, 0.1805257350206375, -0.7531452178955078, 1.0929193496704102, -0.5085049867630005, 0.1328238844871521, 0.8190648555755615, -0.6232591271400452, -0.15927967429161072, -0.1304631531238556, -0.6852604150772095, -0.5693590641021729, 0.0030377681832760572, -0.10590724647045135, 1.1872060298919678, -0.39417579770088196, 0.7843449115753174, 0.0439334399998188, 0.6586803793907166, -1.9034693241119385, 0.6650626063346863, -0.6155534982681274, -0.12239785492420197, -1.0921123027801514, -0.3160383403301239, 0.17604273557662964, 0.033987924456596375, -1.4391980171203613, -0.38026511669158936, -1.5109821557998657, -0.15224036574363708, 0.2254074066877365, 0.13146555423736572, -1.22676682472229, -0.6341312527656555, 1.6170560121536255, 0.6377740502357483, 0.1599024087190628, 0.900079071521759, 0.9764827489852905, 0.9980818629264832, -0.07123997807502747, -1.1266776323318481, 0.2724975049495697, 2.3306734561920166, -0.5920109152793884, 0.0015250886790454388, -1.8946946859359741, 1.6413331031799316, 0.7217535376548767, -0.4071830213069916, 0.04140081629157066, 1.748720645904541, -1.1336654424667358, 1.112098217010498, -0.6156976819038391, 1.7772314548492432, 1.3499691486358643, 0.005566298495978117, -0.1498747617006302, 0.9855784177780151, -0.04305272921919823, -0.12739013135433197, 0.1601867824792862, 1.4542186260223389, 1.3496880531311035, -0.44107145071029663, -0.38018035888671875, 0.7676887512207031, -0.01605718769133091, 0.09846198558807373, -0.9330450892448425, 2.4818220138549805, -0.5926461219787598, -1.6805802583694458, 0.8582903146743774, -0.44476518034935, -1.2165435552597046, -0.33359187841415405, -0.13281367719173431, 1.435949444770813, -0.3898281455039978, -0.8864539265632629, -0.33538007736206055, 0.5175744891166687, 0.2894972264766693, 0.4616963565349579, 1.516180157661438, 0.2344672828912735, 0.6638463735580444, -0.44844821095466614, -0.41782277822494507, 0.7126848697662354, -0.9591561555862427, -0.4655635356903076, -0.5598804354667664, 0.2854486107826233, -1.329380750656128, -2.5922250747680664, -0.501878559589386, 1.046372890472412, 0.5693307518959045, -0.19954822957515717, 2.8152196407318115, 1.1883795261383057, -0.5207405090332031, 1.054823398590088, -0.3182497024536133, -0.28063714504241943, -0.7034648060798645, 0.04001883417367935, 0.58526611328125, -0.7518235445022583, 1.8229764699935913, -0.5208430290222168, 1.85810124874115, 1.3426480293273926, 1.1605844497680664, -0.6947593092918396, 0.4847628176212311, 1.3405295610427856, -0.4858182966709137, -0.4861760437488556, -0.5555896759033203, -0.8250694870948792, 0.797107458114624, -0.16393886506557465, -0.8245609998703003, -0.7140118479728699, 0.03782856836915016, 0.4011646807193756, 0.2257462590932846, 0.6598902344703674, -1.2557497024536133, -1.0353221893310547, -0.25827956199645996, -0.24822361767292023, 0.44708892703056335, 1.1634917259216309, 0.2889060378074646, 0.3911585807800293, -1.2431830167770386, -0.8977581262588501, -1.387528419494629, 0.6087923645973206, 0.16042092442512512, 1.283097505569458, -1.6701465845108032, 0.3731452226638794, 0.5760478973388672, 1.7970924377441406, 0.10614661872386932, -1.8079363107681274, -0.0368206650018692, 0.7398309111595154, 0.3939145505428314, -0.12941499054431915, -0.835548996925354, -0.8618181347846985, 0.0009040456498041749, -0.16905881464481354, 0.018529783934354782, -0.2149893194437027, 0.18811462819576263, -0.8153354525566101, -0.886027455329895, 0.7050787806510925, -0.9382956624031067, -1.2418943643569946, -1.3262529373168945, 0.5258147120475769, -1.2875458002090454, 0.9416368007659912, -1.471964716911316, 0.31243085861206055, -2.1598262786865234, 1.088050127029419, -1.8386400938034058, 0.3532716929912567, 1.1111184358596802, -0.091412752866745, -0.7385249733924866, 0.5841410756111145, -1.4129194021224976, 0.7492080330848694, -1.162341833114624, 0.2551209330558777, -0.004227863159030676, 0.3421773612499237, -0.38784098625183105, 1.3361035585403442, 0.8652714490890503, 2.507551431655884, 0.3871159553527832, -0.34056687355041504, -0.5121262073516846, 0.12653709948062897, -0.9414493441581726, -0.4554964303970337, -0.11574243009090424, -0.7472425699234009, 1.7060102224349976, -1.4385778903961182, 1.052322506904602, 1.2380907535552979, -0.47066447138786316, 0.03783724084496498, 0.16537098586559296, -1.7391217947006226, 1.0441426038742065, 0.4756246507167816, 0.4958489239215851, 1.1481812000274658, -0.9948521852493286, 0.13180580735206604, 1.693067193031311, -0.228326678276062, 0.386530339717865, -0.7389658093452454, -0.7298339605331421, 0.560094952583313, 0.42472344636917114, 0.11577852070331573, -0.9830037951469421, -0.12210529297590256, 0.6858499050140381, 0.15200918912887573, -0.22704315185546875, 0.06659679114818573, -0.35554420948028564, 0.040188152343034744, 1.3377678394317627, 0.23811684548854828, -0.34945148229599, 0.4578971862792969, -1.8829373121261597, 2.3768022060394287, 0.7402709722518921, 1.1368341445922852, -0.6378495693206787, -0.9120972156524658, -1.7372196912765503, -0.48733165860176086, 1.4076162576675415, -0.24508780241012573, -0.026872538030147552, 0.4810009300708771, 0.5188799500465393, 0.7452979683876038, -0.9734580516815186, 0.41394972801208496, 0.05409626290202141, 0.4130503833293915, -0.24021156132221222, 0.6378932595252991, 0.45913222432136536, 1.1446332931518555, -1.7585210800170898, 1.3342702388763428, -0.7560015916824341, 1.8233568668365479, 0.6180092096328735, -0.6080882549285889, 0.4152044653892517, -0.46008655428886414, 1.2756694555282593, -0.8006356358528137, 0.48300695419311523, -0.03169070556759834, 0.5301281213760376, 0.7545537948608398, -0.6207103729248047, -0.2084398865699768, 0.04573024809360504, -0.5146142840385437, -1.2546888589859009, -0.5415309071540833, 0.1805609166622162, 1.0524555444717407, -1.3861868381500244, 0.004956090357154608, 0.46306851506233215, -0.20634716749191284, -1.5209171772003174, 0.6417900919914246, -0.8314117789268494, 1.1875286102294922, -0.541119396686554, 1.0874102115631104, -0.5346945524215698, -0.4315721392631531, -0.9597031474113464, 0.24757567048072815, 0.06250983476638794, -0.40423548221588135, 0.19541020691394806, 0.1460413932800293, 0.3393431007862091, 3.2447924613952637, 0.04120355471968651, 1.2113332748413086, 1.3755955696105957, -0.7566915154457092, 0.7928443551063538, -1.2471559047698975, 0.7911145091056824, 0.11717738211154938, -0.019680136814713478, 0.03642691671848297, 0.9138860702514648, 0.11957956105470657, -1.263323426246643, -2.3463921546936035, -1.3903796672821045, 1.0089136362075806, 0.730535626411438, -1.4542086124420166, -0.5655553936958313, -0.1191616877913475, 0.2961874008178711, 0.8839934468269348, 1.2137084007263184, 0.3275669515132904, -0.3701428771018982, -1.7087368965148926, -0.029496343806385994, -0.9176292419433594, 2.3659708499908447, 0.10767198354005814, 1.3231488466262817, -0.7125160098075867, 0.16361789405345917, 0.4712860584259033, 0.283122181892395, 1.070841908454895, 0.880334734916687, 0.17300401628017426, -2.2315495014190674, -1.0011087656021118, 0.28470203280448914, 2.0220389366149902, 0.2698533236980438, -0.012812329456210136, 0.2181081920862198, -0.5893643498420715, -1.2847769260406494, 0.6690179109573364, 2.1290271282196045, 0.7570310831069946, -0.20506368577480316, -1.885870337486267, -2.0085058212280273, -0.11422654986381531, 1.3773384094238281, -0.7595535516738892, 0.1889694184064865, 0.4592381715774536, 0.7190488576889038, 0.723892092704773, -0.1945309042930603, 0.3794650435447693, 0.16005238890647888, 0.6466169357299805, 1.3776686191558838, 0.3797825574874878, -1.4070605039596558, 1.0481709241867065, -1.4553278684616089, 1.749863624572754, -1.2418689727783203, -0.6236555576324463, -3.081221580505371, 0.914100170135498, -2.976081609725952, -1.66457998752594, -1.207083821296692, -0.09214001893997192, 1.3632709980010986, -0.952025830745697, 1.3029688596725464, -1.6486676931381226, -0.060615863651037216, -0.48276636004447937, -0.9721046090126038, 0.5643975138664246, -1.000756025314331, -1.6395771503448486, -0.41048184037208557, -0.8060283660888672, 0.971860408782959, 1.1959480047225952, 1.6185230016708374, 0.7790283560752869, -0.29505985975265503, -0.4760834276676178, 0.42886826395988464, -0.8597031831741333, 1.3278274536132812, -0.14802813529968262, -2.58164119720459, -1.722995400428772, -1.0217323303222656, -0.23778247833251953, 2.5993776321411133, -1.3186533451080322, -1.5962016582489014, -0.2813710570335388, 0.4917319118976593, -1.407616376876831, -0.6327382922172546, -0.760726273059845, 1.7404351234436035, 1.3262807130813599, -0.05580204352736473, -0.49127209186553955, 0.8276790976524353, 0.8143211603164673, 0.2761780619621277, 0.6705710291862488, -0.17996616661548615, 1.1107730865478516, 1.5205355882644653, 1.2038168907165527, -0.046214137226343155, -2.016914129257202, -2.1077051162719727, 1.4076045751571655, 0.15123175084590912, 0.6386747360229492, 0.46500635147094727, -2.2219491004943848, -1.1467376947402954, -2.0222976207733154, -0.22975239157676697, -0.591442346572876, 1.6126277446746826, 1.722217321395874, -0.7096395492553711, 1.3940708637237549, -1.0230811834335327, 0.28557831048965454, 0.3620545268058777, 0.203158438205719, 0.4179467558860779, 2.3219375610351562, 0.16786861419677734, 0.7075473666191101, -0.4659877419471741, 0.3867467939853668, 0.012636370956897736, 0.8356941342353821, 0.21472126245498657, 0.5852378010749817, -0.7334058284759521, -0.10890444368124008, -0.6227424740791321, 0.5301765203475952, -0.6073393821716309, 1.1450213193893433, -0.44039613008499146, 1.5826605558395386, -0.5615180730819702, 0.730052649974823, 0.26212817430496216, 0.18609899282455444, 0.37593725323677063, 0.03951842710375786, 0.36247506737709045, -0.4909490942955017, -0.8045926690101624, -0.4788852334022522, -0.9259344339370728, -0.2690863311290741, 1.2481032609939575, 1.4205073118209839, 0.14550521969795227, -0.8317387104034424, -1.5817749500274658, -0.18395814299583435, -0.22949253022670746, 0.11036714911460876, -0.033104971051216125, -0.9830533862113953, -0.0067227608524262905, -0.19607844948768616, 0.7794730067253113, 0.413079172372818, -1.1590839624404907, -0.087365061044693, -0.8884400129318237, 0.6126483082771301, 0.05447223410010338, -0.21254512667655945, -0.4375600516796112, 0.7491121888160706, 0.055241771042346954, 1.3981164693832397, 0.22332444787025452, 1.1412088871002197, -1.3483437299728394, 1.2057193517684937, 0.45364511013031006, 1.994822382926941, 0.6274608969688416, 0.07048290967941284, -0.0036745108664035797, -1.776347279548645, 1.4435707330703735, 0.15695059299468994, -0.6718555688858032, -0.04738035053014755, 0.5141388773918152, 0.039077144116163254, -1.0857921838760376, 0.18199844658374786, -0.5641134977340698, 0.8698258399963379, -1.03953218460083, -0.19603905081748962, 2.0533502101898193, 0.7419123649597168, 0.572022020816803, 1.1614689826965332, -0.32724469900131226, -0.5859192609786987, 1.069562554359436, 2.162970542907715, -0.026854215189814568, 1.4114952087402344, 0.25454431772232056, -0.7346610426902771, 0.2525270879268646, -1.1332342624664307, 0.014547896571457386, -0.3678895831108093, 2.2578635215759277, -0.47806689143180847, -0.6269162893295288, -0.935247004032135, 0.643395721912384, -0.8328298330307007, 0.1918187141418457, 0.19534584879875183, -1.3385756015777588, 1.9425859451293945, 1.174350380897522, -0.8400689959526062, -0.03083382360637188, 0.6316540241241455, -0.4752417802810669, -1.5361734628677368, 1.9574438333511353, -0.1318252831697464, -1.7004578113555908, 0.9763742089271545, 0.6146899461746216, 0.09836979955434799, 0.3029758632183075, -1.505317211151123, -0.2908233106136322, 0.3975023031234741, -1.2108546495437622, 0.4877515435218811, -2.949847459793091, 0.4569760859012604, 0.3854258954524994, 1.7603379487991333, -2.679417133331299, -0.06589106470346451, -0.32970887422561646, 0.9637016654014587, 0.4058293104171753, -0.6651713848114014, -1.0492066144943237, -0.1707744598388672, -1.4822731018066406, 0.2485721856355667, -0.052502989768981934, -0.9513005018234253, 1.1076774597167969, -0.8408418297767639, 0.6543943285942078, -1.3068233728408813, 1.1404869556427002, 0.666071891784668, -0.30790433287620544, 1.2453023195266724, -0.30726388096809387, 0.49235522747039795, -0.2573501169681549, -1.3119505643844604, -2.5644023418426514, 2.0730652809143066, -0.6846129894256592, 1.166467308998108, -0.5777201056480408, 1.0585105419158936, 1.3307322263717651, 0.7849795818328857, 0.48111796379089355, 0.7577632665634155, 0.6138417720794678, -1.5012564659118652, -0.46228426694869995, 0.8397860527038574, 0.05153243988752365], [-1.6757997274398804, 1.19391667842865, 1.4355815649032593, -2.6517679691314697, -0.21354107558727264, 0.3917723298072815, 0.29858121275901794, 0.09417875856161118, 0.7567412257194519, -0.31583282351493835, 1.2130334377288818, -2.589111566543579, 0.5999845266342163, 1.1024830341339111, 1.6322309970855713, -0.43250328302383423, -0.37826767563819885, -1.0309425592422485, -0.4761832058429718, 0.4480389952659607, -0.23120367527008057, -1.6606864929199219, -2.316636323928833, 0.9739841818809509, -0.33104515075683594, -2.0520830154418945, 0.8383498191833496, -1.3772703409194946, -2.0749311447143555, 1.1151410341262817, 1.9530352354049683, 0.45048126578330994, -1.089513897895813, -1.418812870979309, 2.0949814319610596, 0.8979818224906921, -0.147965669631958, 1.5858386754989624, 0.2610446512699127, -1.5166016817092896, -0.14287979900836945, -0.22251328825950623, -0.42780569195747375, -1.210774540901184, -0.1552148312330246, 1.252316951751709, -0.7728428840637207, 0.7434287071228027, -0.008943812921643257, -0.8120891451835632, -1.1626535654067993, 1.8180800676345825, 0.3271738886833191, 1.089654564857483, -1.338176965713501, 0.8824209570884705, 1.1712993383407593, -1.0587968826293945, 1.0349512100219727, 0.7825263142585754, -0.24011626839637756, -0.551918089389801, 0.1913086324930191, 2.805868625640869, 0.6903682351112366, 0.2707676291465759, -0.7624363303184509, -0.13924548029899597, 1.1377166509628296, -1.1688874959945679, 0.09588193148374557, 1.538356065750122, -0.07120208442211151, 2.111907958984375, -0.2181442528963089, -1.0851871967315674, 1.6045033931732178, -0.680450975894928, -1.3005672693252563, 0.5272392630577087, -0.26507532596588135, -1.1978189945220947, 0.4746932089328766, -2.49513578414917, -0.7239658236503601, 0.35981225967407227, -0.05446368455886841, -0.21097876131534576, -0.15726949274539948, -0.20154047012329102, -0.36020419001579285, 0.9726373553276062, 1.1194692850112915, 0.01255556009709835, -0.2886270582675934, 0.7190532088279724, -0.5699411630630493, -1.684354543685913, -0.5365381240844727, 0.47415482997894287, -1.9551345109939575, -0.8334276676177979, 0.45333540439605713, 1.8622556924819946, 0.43787044286727905, 0.41470426321029663, -0.697844386100769, 1.557511806488037, -0.6544482707977295, 1.666274905204773, -0.624633252620697, 0.21341301500797272, -0.565595269203186, -0.6538395285606384, 0.011067966930568218, -0.014504908584058285, 0.4245588183403015, 0.373481810092926, -0.15578383207321167, -0.1518927812576294, 1.7539241313934326, -0.6008315682411194, -0.09746258705854416, 1.8957387208938599, -1.9999961853027344, -0.6543819308280945, -0.5734617114067078, -0.3394833505153656, -0.8361620903015137, 0.1285717487335205, -0.06106693297624588, 0.7494111657142639, -1.1241334676742554, 0.15227457880973816, -0.1701061725616455, -0.021619083359837532, -0.3809138238430023, 1.6090798377990723, 2.253938674926758, -0.934559166431427, -0.40203091502189636, -0.03790009021759033, -0.13463647663593292, -0.00782894715666771, 0.6908392906188965, -0.11053171753883362, -0.24328722059726715, -0.19903160631656647, -0.6674144864082336, -1.7424676418304443, -0.7716819643974304, 0.4635434150695801, 1.1372421979904175, -0.03122810833156109, 0.8425677418708801, -0.08533477038145065, -0.6947727203369141, 0.323436975479126, 0.02096160128712654, -0.33256104588508606, 1.0004525184631348, 0.4951501488685608, 1.5702126026153564, -0.06452983617782593, 0.7345389723777771, 0.6611976027488708, -0.7733938097953796, 0.06381569802761078, 0.08279123902320862, 0.3215247392654419, 0.2592601776123047, -0.036433447152376175, -1.0021517276763916, -1.85874605178833, 0.48993104696273804, 0.3794107437133789, -1.1478900909423828, -0.37795886397361755, -1.561365008354187, -0.5633587837219238, 0.3785281479358673, 0.5863548517227173, 0.8979602456092834, -0.048587772995233536, -0.7586411833763123, -0.6232683062553406, -0.8056853413581848, -0.24926380813121796, 0.9495095610618591, 1.4286102056503296, -0.9643951058387756, 0.9975833296775818, -0.2538962662220001, 0.3177143633365631, 0.2908235490322113, 1.3378522396087646, 0.6603882312774658, 0.17552629113197327, -0.5163540840148926, -0.7424205541610718, 0.5429153442382812, -1.5617567300796509, 1.2298020124435425, -0.2966609299182892, -0.08912967145442963, -1.2177884578704834, -0.5673187375068665, 0.5517382621765137, -2.0684938430786133, 1.3082929849624634, 1.317044734954834, 1.039915680885315, 0.09796987473964691, -1.309219241142273, 0.21468748152256012, -0.08106579631567001, -1.0436170101165771, -0.4659487307071686, 1.4567946195602417, 0.6543282270431519, -0.44859960675239563, 0.018092019483447075, 0.3300609588623047, 0.6379973888397217, 1.17030930519104, 0.9204104542732239, -1.285167932510376, 0.7216836214065552, 0.9662028551101685, -1.4977957010269165, -1.8001468181610107, -0.9632816314697266, -0.04307008534669876, -0.27439460158348083, 0.1451776772737503, -0.07060298323631287, 1.351319670677185, -0.5054616332054138, 0.9485729932785034, 1.065239667892456, 0.11618661135435104, 1.2462292909622192, 1.2092396020889282, 0.6569948792457581, -1.0973575115203857, 0.6383847594261169, 1.6331743001937866, 0.634772777557373, -1.6195776462554932, 0.105940081179142, 0.3679077923297882, -0.006486698519438505, -1.5532007217407227, 0.6278954744338989, -0.48238176107406616, 1.1210533380508423, -1.8429782390594482, 0.3069990575313568, 0.4767662584781647, 0.29293832182884216, -0.817901074886322, -0.689944326877594, 0.08868158608675003, -0.07115133851766586, -1.0807092189788818, 0.33587950468063354, -0.5341356992721558, 1.354536771774292, -1.8067134618759155, -1.3354204893112183, 0.5306453108787537, -0.32608795166015625, 1.7595125436782837, -0.39688917994499207, 0.9824506044387817, -0.9062062501907349, 0.1959061622619629, -0.8714489936828613, -0.3106013536453247, 0.6207981109619141, -0.8491368889808655, -0.3324704170227051, 0.03498605266213417, -0.22460611164569855, 0.6908884644508362, 0.9155157208442688, 0.18136706948280334, 2.3474977016448975, -2.40787410736084, -0.7740526795387268, 0.4024801552295685, 0.9528842568397522, 1.13151216506958, -0.9855883717536926, -0.8028299808502197, -1.9364815950393677, 0.6071740388870239, 1.2291659116744995, -0.2991054654121399, 1.8537214994430542, -0.9594858288764954, -1.5415927171707153, -0.242253839969635, 0.9326844811439514, -0.7614068984985352, -0.8532376289367676, 0.030225127935409546, 0.396538645029068, 0.1351221650838852, 0.1264037936925888, -1.398425579071045, 0.31231868267059326, 0.7067258358001709, -0.3998071253299713, -1.6297065019607544, -0.06105193868279457, -0.43985626101493835, -0.2720223069190979, -1.0108619928359985, -0.8844300508499146, -0.6133030652999878, -0.4306741952896118, 0.1991598904132843, 1.0667229890823364, 0.07107266038656235, -1.0987229347229004, 0.254703551530838, 0.7340441942214966, 1.3460725545883179, 0.8385520577430725, -0.6653974056243896, 0.18819762766361237, 0.8796291947364807, 0.5347495675086975, 1.3801655769348145, -2.3210158348083496, -0.4151304066181183, -0.3620823621749878, -0.5681772232055664, -0.055563878268003464, 0.2365824282169342, 0.38369086384773254, -0.7707534432411194, 0.6697723269462585, 0.10065717995166779, 0.48200541734695435, -0.5676122903823853, -1.9595390558242798, 1.693820834159851, 0.6708920001983643, -0.6350322365760803, 0.24542836844921112, -0.8170501589775085, -1.6997482776641846, 0.08080580830574036, -1.656803846359253, 1.1002438068389893, 0.11294880509376526, 0.365555077791214, 0.13794739544391632, 0.4419756233692169, 1.2025243043899536, 0.1963634490966797, -1.0428261756896973, 0.16703133285045624, -0.06263147294521332, -1.1694928407669067, -1.115431547164917, 0.4818006753921509, -1.0897573232650757, 0.11739514768123627, 0.45569273829460144, 1.4427447319030762, -0.9321556687355042, 0.0380629301071167, -0.3671870231628418, -0.27707037329673767, 0.6850987672805786, 0.46413126587867737, -0.3105458915233612, 0.3563404679298401, 0.29463833570480347, -0.3290759325027466, 0.675482451915741, 0.26796793937683105, -0.08554386347532272, 0.046466484665870667, -1.0060168504714966, 0.29871752858161926, 0.1569061577320099, -0.2269834578037262, -0.07017859816551208, 1.3208792209625244, -1.0250003337860107, 0.8765905499458313, 1.1480973958969116, 1.051591396331787, 1.6423027515411377, 1.562447190284729, -0.8075600266456604, -0.43955758213996887, 0.6058319807052612, -0.22278600931167603, 1.3308438062667847, -0.33680713176727295, 0.8181517720222473, -0.22211475670337677, 0.0768924355506897, 0.4166342616081238, -0.3261462152004242, -1.619510531425476, 0.2618183493614197, -1.5436270236968994, 0.9831802248954773, -0.5996472239494324, -0.9579231142997742, 0.12172795087099075, 2.079831123352051, 0.4297931492328644, 0.2546008229255676, 0.864012598991394, -0.085350401699543, 0.25252339243888855, 0.8748433589935303, -1.1844968795776367, -0.1920279860496521, 0.6627631783485413, -0.4490077495574951, -0.7900625467300415, -0.582021951675415, -0.24398456513881683, -0.8420648574829102, 0.5471869111061096, 0.843880832195282, -0.701553225517273, -0.28332048654556274, -0.42904266715049744, -1.1670897006988525, -0.4121048152446747, 0.787705659866333, 1.4420989751815796, -1.3578784465789795, -1.0384364128112793, -0.3076939880847931, 0.9998602271080017, -0.6382137537002563, -0.7687551379203796, -0.6835666298866272, 1.0377283096313477, -0.8372746706008911, 0.9733082056045532, -0.9800434112548828, 0.3110555112361908, -0.31680575013160706, 0.7883464097976685, -1.279789686203003, -1.3973300457000732, -0.6263348460197449, 0.04170258715748787, -1.6247777938842773, 0.2566882073879242, 0.30831825733184814, -2.439779758453369, -0.9223287105560303, -1.0033893585205078, -0.09650292992591858, 1.463387370109558, -2.4081919193267822, -1.8419822454452515, -0.16836725175380707, 1.771237850189209, 0.688240110874176, 2.178340435028076, 0.4359101355075836, -0.7764719724655151, 2.007765054702759, 0.7003947496414185, -0.46306782960891724, -0.17709980905056, -0.29291433095932007, -0.20142865180969238, 1.2395820617675781, 1.0820749998092651, -0.43186578154563904, -1.8573205471038818, 0.875667929649353, 0.20741446316242218, -0.28762561082839966, 0.3796588182449341, -0.7392385005950928, -0.012267697602510452, -0.2824901342391968, -0.3989371359348297, 0.38141706585884094, -0.45707279443740845, -0.33345651626586914, 1.8108487129211426, -0.42327407002449036, 1.557287335395813, -3.3329646587371826, 1.3527848720550537, -1.271676778793335, 0.17903566360473633, 0.34827375411987305, 0.11405441164970398, 1.8870395421981812, 0.5882271528244019, -0.3340049982070923, 1.2796423435211182, 0.6645381450653076, -0.49493756890296936, 0.6197465658187866, 0.032403018325567245, -0.8434267044067383, -0.5816682577133179, -0.5586920976638794, 0.12581780552864075, 0.08143085241317749, 0.3017905056476593, -0.3781375586986542, -0.3159730136394501, 0.05757804960012436, 1.2049463987350464, 1.9120138883590698, -0.9192541241645813, -0.1408657282590866, -1.597878336906433, 1.1123429536819458, -0.3268098533153534, 0.47796204686164856, 0.35132625699043274, 0.48740658164024353, 1.461356282234192, 1.0547459125518799, 1.7249462604522705, -0.32601162791252136, 0.06450650095939636, 1.4518932104110718, 1.085937738418579, -0.2670811712741852, -0.08949540555477142, -2.2561511993408203, -0.903710126876831, -0.9344903826713562, 1.8843458890914917, -1.505491018295288, 0.3033662736415863, 0.9195634722709656, -1.795109748840332, -0.8589303493499756, 1.1058903932571411, -3.37682843208313, -0.6562143564224243, 0.3491222560405731, 0.6840128898620605, -2.1553730964660645, 2.2039260864257812, 0.9783051013946533, 0.26470550894737244, 0.8115668296813965, -0.6271917223930359, -0.49765750765800476, 0.8249881863594055, 0.7019772529602051, 0.19824549555778503, 0.5128478407859802, -0.5281427502632141, -0.12627863883972168, 2.024657964706421, 0.06257229298353195, 1.055870771408081, -1.369407057762146, -0.129968523979187, -2.0004656314849854, 0.0325247123837471, 0.31564101576805115, -0.5296838879585266, -1.7408881187438965, 1.0017998218536377, 1.4378606081008911, -0.16570603847503662, 1.0038108825683594, -1.00467050075531, -0.3522997200489044, 0.43141981959342957, -1.1123579740524292, -0.04804791882634163, -0.7473843097686768, 0.12120618671178818, 0.35437148809432983, 0.8134772777557373, 1.2572137117385864, 0.09681712836027145, 0.2115994542837143, 0.8699548840522766, 1.3737127780914307, -0.607753336429596, -0.8279002904891968, 0.08336293697357178, -1.9445304870605469, -1.119752049446106, -1.300379991531372, 0.5167135000228882, -1.6896573305130005, -0.4759998321533203, 1.9402539730072021, -0.49659523367881775, -0.45597273111343384, -1.2747589349746704, 0.7167180776596069, 0.0301365926861763, -0.4630517363548279, 1.4495290517807007, 2.000656843185425, -2.4839587211608887, -1.0787508487701416, 0.6739335656166077, -1.208116054534912, 1.578648567199707, -0.6544853448867798, -0.6409561038017273, -0.048185016959905624, 1.8233176469802856, -0.20468495786190033, -0.08380158990621567, 0.21254324913024902, 0.4348852336406708, -0.6404320597648621, 0.47672370076179504, 0.7260376214981079, -0.30819588899612427, 0.8181971311569214, -0.5353663563728333, 0.6901938915252686, -2.0928568840026855, 0.10324740409851074, 1.1657626628875732, -0.05720199644565582, -0.4212172031402588, -1.540435552597046, -2.260612726211548, -0.3869863450527191, -0.04639718309044838, 0.026011137291789055, -2.2733588218688965, -0.635036826133728, -0.36053669452667236, -0.16747106611728668, 1.0528435707092285, 1.0160402059555054, -1.2103004455566406, -0.8563247919082642, -2.32928466796875, 0.8271171450614929, -1.0191246271133423, -0.0211032722145319, -0.9273120760917664, 1.1957319974899292, 1.8175859451293945, -0.8069272041320801, -1.9425053596496582, -0.6076056957244873, 1.4843806028366089, -0.6998313665390015, 0.48879095911979675, 1.125590443611145, 0.6723503470420837, -0.7638671398162842, -0.38134628534317017, 0.6961270570755005, 0.0878019705414772, 0.5385744571685791, 1.3499500751495361, -0.6345972418785095, -0.22657252848148346, -1.6280465126037598, 1.048248529434204, -0.9219000935554504, -0.14200641214847565, 1.4093326330184937, 0.0790395438671112, -0.1925039440393448, -1.0557624101638794, -0.8326428532600403, -0.6575345993041992, -0.8889046907424927, -1.524836778640747, 0.1430005580186844, -1.232146978378296, -0.4701630473136902, 2.5144052505493164, 0.33736634254455566, -0.22558118402957916, -0.5834994912147522, 0.3695294260978699, 0.5282397866249084, -0.15546342730522156, -1.2535061836242676, -0.4899517297744751, -0.2855512499809265, 0.40878838300704956, 0.6232917308807373, 2.3375344276428223, -1.5342826843261719, -1.301033616065979, -0.03194153308868408, 0.6688677668571472, -1.2510204315185547, -2.046898126602173, 2.5684006214141846, 0.5220775604248047, 0.3398130238056183, 0.6786259412765503, -0.9562017321586609, 0.5188602805137634, 0.47480931878089905, -1.5998131036758423, 0.21943043172359467, 1.9948956966400146, -0.07593348622322083, 0.3218541741371155, -0.7780285477638245, -0.21442434191703796, 0.8624813556671143, -0.7030848264694214, 1.6474812030792236, -1.897823691368103, 0.6849251985549927, 1.6493593454360962, 1.1316946744918823, -1.684058666229248, 0.8537360429763794, -0.1642277091741562, 1.5950206518173218, 0.9653070569038391, -0.8093590140342712, 0.6661990284919739, -0.7384271621704102, 1.4686839580535889, 0.5904415845870972, 0.28277671337127686, 2.0551564693450928, -1.0678834915161133, 0.9758438467979431, 0.2520541250705719, -1.0761728286743164, -0.011101314797997475, -0.9374300241470337, 0.15549466013908386, 0.4981379508972168, 1.1977388858795166, 0.5310824513435364, -0.4584069550037384, 0.4959355592727661, 1.374348759651184, -0.5302586555480957, 1.1658183336257935, -1.9577525854110718, 0.2867511808872223, 1.1062824726104736, -0.9935383796691895, -1.0715060234069824, 0.6803781390190125, -0.03711960092186928, 1.8560056686401367, 1.3394420146942139, -0.680107057094574, -1.5214697122573853, -0.23368580639362335, -0.9748002886772156, 1.0002740621566772, 0.8391481637954712, 1.3839725255966187, -1.9762839078903198, -0.580467939376831, -0.7962254285812378, 0.9534736275672913, -0.03976203128695488, 0.5999062657356262, -1.544742226600647, -1.8415422439575195, -1.8068156242370605, 1.4920405149459839, 0.3804571330547333, -1.9236036539077759, 0.788600742816925, 1.1385124921798706, -0.07671002298593521, 0.08696086704730988, -0.3743252456188202, 2.469794273376465, 0.9068237543106079, -0.06878989189863205, 1.3488763570785522, 0.516903817653656, 1.053495168685913, 1.4301270246505737, 0.867647111415863, 0.5142684578895569, 0.3918084502220154, -1.3770318031311035, 0.23340412974357605, 0.03145495429635048, 0.732817530632019, -0.8198927640914917, 1.2378044128417969, 0.9441249370574951, -0.3115468919277191, 0.7170336842536926, -0.19350175559520721, -0.9644865989685059, 0.3763505518436432, 1.0653189420700073, 0.9753127694129944, 0.7286346554756165, 0.5251291990280151, -1.8798528909683228, -0.5804377794265747, -1.972692608833313, -0.9770854115486145, 0.20552417635917664, -0.9925632476806641, -0.6064180135726929, 0.6101366281509399, 1.7203941345214844, -0.0031021079048514366, 0.7341369390487671, 0.29625648260116577, 1.0672504901885986, -0.05695774406194687, 0.9266620874404907, -1.1908546686172485, 1.7319462299346924, 1.622650146484375, -1.1729085445404053, 0.5935592651367188, -0.8512584567070007, -0.7954720854759216, 0.9639852046966553, 0.8904696106910706, -0.5743362903594971, -0.38058552145957947, -0.4748474657535553, -0.6513147354125977, 1.8228113651275635, 0.1491091251373291, 0.035730719566345215, -1.0683437585830688, -0.696043074131012, -1.0806679725646973, 1.7951596975326538, -0.20315125584602356, 0.1276932954788208, 0.6946530938148499, -0.449808806180954, 2.1371548175811768, 0.284013032913208, -1.230280876159668, -1.8950186967849731, 0.2450079619884491, -2.2079622745513916, 2.5701961517333984, 1.4911863803863525, 1.2504801750183105, 1.047369122505188, 1.3952572345733643, -0.40265703201293945, 0.16201530396938324, -2.051023006439209, 0.46808549761772156, 2.3309695720672607, -0.29199761152267456, -0.018014146015048027, -0.8741795420646667, 0.21580494940280914, -0.1744118332862854, 0.3936285674571991, 0.4590325653553009, 0.12639178335666656, -0.5492831468582153, 1.2044297456741333, -0.6900274157524109, 0.06821732223033905, -0.2562243938446045, 0.4305417835712433, -1.2401269674301147, 0.30614981055259705, -0.6313219666481018, 1.5407147407531738, 1.5875180959701538, -0.44232407212257385, -0.17501451075077057, -0.6139010787010193, -2.530484914779663, -0.7094313502311707, -1.8754020929336548, 2.1808345317840576, -1.4481860399246216, 0.11834307760000229, 0.8196903467178345, 0.6476742625236511, 0.6462891697883606, -2.4152157306671143, -0.10568472743034363, 1.4726111888885498, -0.16993248462677002, 0.3395124673843384, 0.6284653544425964, 0.34443002939224243, 0.2904086410999298, -1.743214726448059, -1.7153903245925903, 0.6524388790130615, 1.4055877923965454, -1.0419832468032837, 1.435704231262207, -0.7117937207221985, -1.618500828742981, -0.8372708559036255, -2.059846878051758, 1.0666197538375854, -0.48434117436408997, 0.8356099724769592, -0.34251463413238525, -0.7039322257041931, 0.4268609285354614, 0.6680664420127869, 0.8782896399497986, 1.6942819356918335, 0.3407202661037445, 0.6998454928398132, -0.29063743352890015, 0.18002671003341675, 3.4815213680267334, -1.129638433456421, -0.3003145754337311, 0.2745979428291321, -0.7037837505340576, -1.7405483722686768, -0.38824760913848877, 1.9098663330078125, -0.1883707195520401, 0.5069028735160828, 1.521164894104004, 0.13030582666397095, -0.0992298349738121, 1.5793389081954956, -0.5477482080459595, -1.13188898563385, 0.3352183699607849, -1.330479621887207, 1.2520848512649536, -0.40021660923957825, -0.2063770592212677, -0.6636419296264648, -0.7007955312728882, 0.27537935972213745, -0.32872477173805237, 0.10725415498018265, -0.7070756554603577, 0.08280395716428757, -0.7023904919624329, 0.10143013298511505, 0.1389942616224289, -1.8433952331542969, 0.6140623092651367, -1.8627841472625732, 0.6051667928695679, 1.1319149732589722, 1.3882718086242676, -0.3346903324127197, 0.39900723099708557, -0.8680940270423889, -0.15868481993675232, 1.950434923171997, 0.17137491703033447, 0.530454695224762, 1.0354864597320557, -0.9586918354034424, -1.487393856048584, -0.17098593711853027, 0.39702919125556946, -0.6037843823432922, 0.5884247422218323, 0.3741753399372101, -0.8551217913627625, -0.6703858971595764, 0.14696308970451355, -0.8802511692047119, 1.1764622926712036, -1.1321555376052856, -0.851824164390564, 1.2351189851760864, -0.4590164124965668, 1.1486040353775024, -0.724477231502533, 0.9087978005409241, -0.19386348128318787, 0.13080142438411713, 0.03782776743173599, -1.1063350439071655, -0.6406762003898621, -0.007102507632225752, 1.1725341081619263, -1.3269046545028687, 0.3384447395801544, -0.4003363847732544, -1.0193259716033936, 0.6428554058074951, -0.17550665140151978, -1.7082303762435913, -0.8057296276092529, -0.16185130178928375, 1.140108585357666, 1.3517793416976929, 0.48386022448539734, 1.2531657218933105, 1.2899670600891113, 0.8068121671676636, 1.349631428718567, -0.19023524224758148, -0.43833261728286743, 0.3867760896682739, 1.0483609437942505], [-2.4781582355499268, 0.0629793107509613, 2.087885618209839, -1.4721870422363281, -0.8428189754486084, -0.8466094136238098, 0.46447762846946716, -1.8970896005630493, 0.23550041019916534, 0.963710606098175, 0.7356802821159363, 0.6623421311378479, 1.4247190952301025, -0.305119127035141, -0.6990529298782349, -1.7295362949371338, -1.0743286609649658, -0.5961979031562805, 0.9637941718101501, 0.5434146523475647, -1.6481549739837646, 1.2536598443984985, -0.3129826784133911, -0.9839249849319458, 0.3338189721107483, -2.0531346797943115, -0.4615398347377777, -0.3882145285606384, -0.048205580562353134, -0.18146024644374847, -0.3867362141609192, -2.1993091106414795, 0.41722699999809265, 0.42453449964523315, -0.9376814365386963, -1.1694090366363525, 0.23607663810253143, 0.638304591178894, -0.9042914509773254, -0.7980204820632935, 0.11520396918058395, 0.28641146421432495, 0.7631726861000061, -0.39460089802742004, -0.32273104786872864, -0.44680801033973694, 0.8698744177818298, -2.1889426708221436, -0.8571179509162903, -1.1231367588043213, -0.2238350361585617, 0.25955629348754883, -1.3660699129104614, -0.13434870541095734, 0.5427293181419373, -0.3000289499759674, 0.5729917883872986, -0.14709562063217163, -0.7074474692344666, 0.796762228012085, -0.2646859884262085, 0.09898493438959122, -0.1570408046245575, 0.003440598025918007, -1.428661584854126, -0.1717870980501175, -0.8275348544120789, -0.7994363903999329, -0.2352977842092514, 0.9856599569320679, 0.054517678916454315, -2.2382729053497314, 0.3820124566555023, 0.6550632119178772, -0.6013915538787842, 0.3556102514266968, 1.349625825881958, -0.497103214263916, -0.23625577986240387, 1.38093900680542, -1.1407766342163086, 0.77946537733078, -0.22682435810565948, -1.657652497291565, -0.7292972207069397, 0.08255109190940857, -0.3355255722999573, -0.17893438041210175, 0.4075595438480377, -0.4122512936592102, -1.3545269966125488, 0.9446581602096558, 0.5788482427597046, -1.0048002004623413, 1.0609183311462402, 0.6026571989059448, -0.8791567087173462, -1.2867363691329956, 0.835029661655426, 0.4895147383213043, 0.6171641945838928, 0.24953509867191315, -0.006009745877236128, 0.7229946255683899, -0.5225152969360352, 0.532848060131073, -0.3230953812599182, 2.016448974609375, 1.4437369108200073, 0.12223351746797562, -0.20373474061489105, 1.8420343399047852, -1.9423518180847168, 0.652530312538147, 0.4385514557361603, -0.5861511826515198, -0.12215826660394669, 2.622313976287842, 0.0033113518729805946, -0.3588840067386627, 0.8647708296775818, -0.40982532501220703, 0.0031096204183995724, -0.3144526779651642, 0.8566187620162964, -1.9456206560134888, 1.6808829307556152, -0.030842049047350883, 1.243901252746582, 1.5335451364517212, -0.3092448115348816, 0.4577685594558716, -0.9271381497383118, -0.9273452162742615, -0.15340375900268555, -1.938957929611206, -0.8338293433189392, -0.4753500521183014, -0.15171867609024048, -1.0681004524230957, -0.3838300406932831, -0.6999067068099976, -2.1727025508880615, 0.7722064852714539, 0.8982803821563721, 0.15996123850345612, -0.34868916869163513, -2.19234561920166, 0.8190810084342957, 0.9334656000137329, -0.19270606338977814, 1.30509614944458, -1.0510839223861694, 2.1746373176574707, -0.8215872645378113, -0.7831440567970276, 1.1615897417068481, 0.905844509601593, -0.9713966250419617, -0.01892891898751259, -0.7186183333396912, -0.06141570210456848, -1.4652235507965088, 1.3773759603500366, -0.11481811851263046, 1.3746833801269531, 0.26443037390708923, 1.0712403059005737, 0.4462975561618805, 0.8698557019233704, 0.9207957983016968, 0.4309404194355011, -0.5863932371139526, 1.0690773725509644, -1.065247893333435, 0.5657800436019897, -0.9593032598495483, 0.6764125823974609, -2.2954697608947754, -0.9000158309936523, 0.6358206272125244, -1.556899905204773, 0.4307638108730316, 0.3014525771141052, 1.9880237579345703, 0.5269145965576172, 0.7077068090438843, 1.3839828968048096, -0.2809199094772339, 1.3908543586730957, -0.5031052827835083, -0.3203378915786743, 1.115513563156128, -0.7977241277694702, -0.21312518417835236, -1.617144227027893, -0.25568491220474243, 0.2084057778120041, 0.38769444823265076, -1.1051474809646606, 1.5121655464172363, 0.7440279126167297, -0.7670692801475525, 1.5883101224899292, -1.0517319440841675, -0.12849126756191254, -0.43856140971183777, -0.4336727559566498, 0.21703079342842102, 1.027773380279541, -0.2614736258983612, 0.131600022315979, -0.9197073578834534, 2.234894275665283, 1.9299920797348022, 1.1711353063583374, -0.36024922132492065, 0.9521777033805847, 0.047715503722429276, 1.0504651069641113, -0.2638396918773651, 1.5920748710632324, -0.16051331162452698, -1.2332899570465088, 0.14745570719242096, 0.11381881684064865, 0.3319224417209625, 1.0165064334869385, -0.8485785126686096, -0.08638456463813782, 1.1774287223815918, -0.24906869232654572, 0.5813988447189331, -0.12157157808542252, 0.7006220817565918, -1.8848822116851807, -0.5650244355201721, 0.7652512788772583, 0.8847956657409668, -0.051652777940034866, 0.7662980556488037, -1.1829952001571655, 0.9124863743782043, 0.589870274066925, -0.3013645112514496, -1.0396851301193237, -0.11375220119953156, 1.2834194898605347, 0.8082130551338196, -0.8260651230812073, 0.5747863054275513, -0.6181987524032593, -0.7564892172813416, -0.9181119203567505, 0.04086993262171745, 1.7169418334960938, 0.6736518144607544, 0.0490473248064518, 0.22170734405517578, -0.15377528965473175, -0.10624508559703827, -0.43583062291145325, 0.7760590314865112, 2.365267515182495, -0.31391847133636475, 0.8563953042030334, 1.0702834129333496, -0.0780353844165802, -0.6403265595436096, 1.4527579545974731, 0.12481902539730072, 1.4474005699157715, 1.2840468883514404, 0.6337282657623291, 0.5442861914634705, 0.4370465576648712, 0.75175940990448, -0.659986674785614, 0.9516708254814148, 0.058727290481328964, -1.2310634851455688, 0.27610981464385986, -0.9734647274017334, -0.5275446176528931, 0.14415213465690613, 0.11724039912223816, -0.9169998168945312, -0.6179970502853394, -2.097611665725708, 0.12627997994422913, -1.0532784461975098, 0.4007391035556793, 0.5298435091972351, 0.6071152687072754, -0.2526795566082001, 1.4321342706680298, -0.6095370650291443, -1.9057902097702026, 0.7256649732589722, -0.16365739703178406, -1.3983865976333618, 0.35090935230255127, -1.0398237705230713, -0.011813074350357056, -0.3712942898273468, 0.8380367159843445, 0.9313662052154541, 1.2000945806503296, 0.5581309795379639, 0.1519315242767334, -0.49166014790534973, -1.124815821647644, 1.2270506620407104, -1.4725159406661987, 1.5308232307434082, 1.3002843856811523, 0.025713803246617317, -0.6280592679977417, -0.4653138220310211, -2.2157540321350098, -1.2363072633743286, 0.6747546195983887, 0.5268824696540833, 1.5959059000015259, -0.3831164240837097, -1.3637834787368774, -0.4671575129032135, 0.4710443317890167, -0.5728887915611267, 0.19154776632785797, -0.5851451754570007, 0.5848417282104492, 0.420815646648407, -1.9815735816955566, 0.46167537569999695, 0.17163348197937012, 1.3939300775527954, 0.8276792168617249, -2.0554957389831543, -0.6891396045684814, -0.8875438570976257, -1.220314860343933, 0.2674282491207123, 0.9151139855384827, -1.5801891088485718, 0.0010930063435807824, -2.180281400680542, -0.3154867887496948, 0.064548559486866, 0.4078817069530487, 0.32886239886283875, 0.2701602280139923, 0.6364260911941528, -0.5885716080665588, 0.9283584356307983, 0.8049116730690002, 0.5447239875793457, 0.24128498136997223, -1.4697424173355103, -0.07892028242349625, -1.9338278770446777, 0.6367636919021606, 1.7224047183990479, 0.7962133884429932, -0.45248737931251526, -0.609188973903656, -0.13445785641670227, 1.3213411569595337, 1.1838493347167969, -1.1653109788894653, 0.9340725541114807, -0.7456897497177124, 1.44242525100708, -0.04191649332642555, -0.8414148688316345, 1.1087228059768677, 0.09518467634916306, -0.869814932346344, -1.510629415512085, -2.6448051929473877, 0.6219772100448608, -0.1451261043548584, -0.14667315781116486, -1.4933412075042725, 0.49320971965789795, 0.8244261145591736, -0.8864638209342957, 0.4506947100162506, -0.4150144159793854, -0.6016107201576233, -1.2914890050888062, -0.2541162073612213, 1.0754990577697754, -0.24820466339588165, 1.0411547422409058, 0.8602500557899475, 0.18557970225811005, 0.9008534550666809, -0.6444811820983887, 0.03345376253128052, 1.8377182483673096, -0.4938211441040039, -0.8080923557281494, -0.8450437188148499, 0.7132463455200195, -0.6272901296615601, 2.5522611141204834, -0.27768757939338684, 0.3163300156593323, -0.3280077278614044, -0.36886852979660034, 0.7394766211509705, 2.1275131702423096, 0.3444445729255676, -0.2629931569099426, -1.0957759618759155, -0.021929334849119186, 0.5902833342552185, -0.5257787108421326, 0.2535514235496521, 0.322150319814682, -0.7981892824172974, 1.275816798210144, 1.7812079191207886, -0.9821166396141052, 0.5709035992622375, 0.9892975091934204, 0.61944580078125, -0.5508425831794739, -0.9573772549629211, -1.3237910270690918, -1.1967276334762573, -0.7797675132751465, -1.625126600265503, -0.749403178691864, 1.5261667966842651, 0.9460228681564331, 0.19618527591228485, -1.0743441581726074, 1.9462953805923462, 0.786935031414032, -0.8485251069068909, 0.4442443251609802, -0.3803703784942627, -0.34552475810050964, 0.449724942445755, -0.8510299921035767, 0.005099036730825901, 0.47366324067115784, 1.38571035861969, 0.7371813058853149, -0.9052103757858276, -0.47452929615974426, -0.3384842276573181, 0.7272388935089111, -0.36401233077049255, -1.0663886070251465, -2.780780553817749, -0.920539915561676, -0.2560393214225769, 0.7843372225761414, -0.3372076153755188, 0.2603631317615509, -0.6852415204048157, 0.2810920178890228, 0.005648477002978325, -0.669116735458374, 1.313788652420044, 0.5284958481788635, 1.1839802265167236, 1.3173047304153442, 0.40963461995124817, 0.8032720685005188, 1.4777064323425293, -1.1185166835784912, 1.8885329961776733, -0.20026426017284393, -0.6510255932807922, 1.2745110988616943, 0.21101395785808563, -0.8914275169372559, 1.2602070569992065, -1.210563063621521, 0.23202906548976898, 2.455850124359131, -0.7698373794555664, 0.5912877321243286, -0.6095247268676758, -0.10629062354564667, -0.2163485884666443, 0.18861506879329681, 0.391230046749115, 1.0863122940063477, -0.4850219488143921, -0.026541460305452347, 0.9576268792152405, -0.6524726748466492, 0.3115396201610565, -1.4234803915023804, -1.0886119604110718, -0.762006402015686, 1.7779875993728638, -0.4182198941707611, 0.8747509717941284, -2.355614423751831, -0.833911657333374, 1.4002786874771118, -0.5549699068069458, 0.3491194546222687, -0.5296610593795776, 0.6398872137069702, -0.3279464840888977, -0.6424254775047302, 1.5569950342178345, 0.41430211067199707, -0.998092532157898, 0.7119418382644653, -1.1911603212356567, 1.119039535522461, -0.2866121232509613, -0.06161224842071533, 0.40184202790260315, 2.4842710494995117, -1.5416722297668457, 0.16408197581768036, -0.49048280715942383, -0.1697208136320114, -1.2136945724487305, 0.5402374267578125, 0.23195835947990417, 0.26050806045532227, -0.13177049160003662, -0.20802195370197296, 0.03522380068898201, -1.1371688842773438, 0.32519689202308655, 1.157880187034607, 0.9148995876312256, -0.9619181752204895, 0.14377163350582123, -0.3763464391231537, 0.07113946974277496, -1.1182912588119507, -0.7286427021026611, -0.3145436942577362, -0.9228381514549255, -1.4737166166305542, 0.6430837512016296, 0.43584662675857544, 0.06738084554672241, -0.8696163892745972, 0.3991275727748871, 0.47177252173423767, 0.22080278396606445, -1.101505160331726, 1.7686033248901367, 1.6232428550720215, 0.29171881079673767, -0.6474523544311523, -0.5043119788169861, 0.10975868999958038, -1.1422090530395508, -0.5706073045730591, 1.2867001295089722, -1.1994158029556274, 0.3257690370082855, -1.1545665264129639, 0.15317556262016296, 0.32078632712364197, -0.7678468227386475, -0.11201279610395432, -1.2647922039031982, 0.5977375507354736, 0.15116068720817566, -0.45684579014778137, 0.5924176573753357, -1.0366370677947998, 0.6954179406166077, -0.6314080953598022, -0.373870313167572, -1.2445285320281982, -0.6782752871513367, 0.7330669164657593, -0.9622489213943481, -1.2478597164154053, -2.254041910171509, 0.8860915303230286, -0.4013119339942932, -0.27189725637435913, 0.5900177359580994, 1.9300538301467896, -1.0238709449768066, 1.8417434692382812, -0.8623035550117493, 0.8864136934280396, 0.138469398021698, -0.11851207911968231, -1.9617831707000732, -0.4311036467552185, 1.3415899276733398, -0.10969799011945724, -0.2674552798271179, -1.0997416973114014, -0.7207704186439514, -0.37398213148117065, -0.228805810213089, 2.1500813961029053, 0.6575329303741455, 0.2740545868873596, -0.7131651043891907, 0.28083786368370056, -0.7421307563781738, 1.5824016332626343, -0.3497779965400696, 0.5715503692626953, 0.014888168312609196, 0.3453722894191742, -0.7647795081138611, -0.03105331026017666, -1.5634849071502686, -0.7645585536956787, -2.076946496963501, -0.797713041305542, 1.413346529006958, -0.5943465828895569, 1.7021284103393555, -0.24243506789207458, -1.6354961395263672, 0.47065669298171997, 1.2308567762374878, -1.6303120851516724, -0.08771836757659912, 0.050810717046260834, -1.810445785522461, 1.1862232685089111, -0.10441476106643677, 0.23787416517734528, -1.3686771392822266, 0.734058678150177, -0.12944521009922028, -0.0036468664184212685, 2.300678014755249, 0.5131282210350037, -0.1088922843337059, 0.11494267731904984, 1.3540047407150269, 0.7509639263153076, -0.3488789200782776, -0.4291399121284485, 1.0725346803665161, -1.0668420791625977, -0.08724494278430939, -0.4156383275985718, -0.5628044605255127, -0.2347155660390854, 1.0509222745895386, 0.03841567784547806, -0.9212390780448914, 0.36772987246513367, 0.7226101756095886, 0.1462937295436859, -0.21527819335460663, 0.025022905319929123, -1.5433974266052246, 0.9649277925491333, 1.0592538118362427, -0.4325280785560608, -1.0059734582901, 0.2035052329301834, 0.3061516284942627, 0.18762291967868805, -1.7304353713989258, -1.4795893430709839, 0.001683291862718761, 0.6612589359283447, -1.2455273866653442, 1.8654274940490723, 0.5008774995803833, -0.7777829766273499, 0.5144192576408386, 1.626543641090393, 0.05274725705385208, 0.20770058035850525, 0.21157778799533844, 2.1152307987213135, 0.1585482656955719, 0.5436667799949646, 0.6553986072540283, 1.1375374794006348, 1.3515211343765259, 1.59478759765625, -0.04435654729604721, 0.018155496567487717, 0.5030937194824219, -0.4333397150039673, -0.7554775476455688, 1.008201003074646, -0.19988946616649628, -1.505092978477478, -0.9897280335426331, 0.4918113052845001, 1.9208577871322632, 0.4805574119091034, -0.5116623640060425, 0.36460140347480774, 0.12660539150238037, 1.4159326553344727, -0.8069347143173218, -0.4648132026195526, -1.3958626985549927, 0.031803250312805176, -0.6995816230773926, 0.1362566202878952, 0.8193976879119873, -0.1966336965560913, -1.130645513534546, 0.41768431663513184, 1.3138420581817627, -0.00626031868159771, 0.971382737159729, 0.6431533694267273, 1.1040401458740234, -0.7262354493141174, -1.4022961854934692, -1.2955636978149414, -0.287038654088974, -0.5177411437034607, -1.0947225093841553, -1.228304386138916, -0.2519831955432892, 1.709113359451294, 0.26205986738204956, -0.15419164299964905, 0.26061317324638367, 0.24566639959812164, 0.6003407835960388, 0.9110100269317627, 0.5939617156982422, -1.3057996034622192, 0.3573985695838928, 1.2768874168395996, -0.8632926940917969, -1.1270079612731934, -0.6324305534362793, -1.7857344150543213, 1.131967306137085, 1.6820998191833496, -0.9046570658683777, 1.6415481567382812, 1.7239699363708496, -0.08919819444417953, 1.0126813650131226, -1.256364345550537, -2.2286293506622314, -0.2274339348077774, 0.43388497829437256, -0.6113142967224121, -0.9053242802619934, 0.6355202794075012, 0.24526150524616241, 1.028461217880249, 2.106034755706787, -0.5731382369995117, -1.612966775894165, -0.7746963500976562, 2.022214889526367, -0.3590548634529114, -0.4767104387283325, -0.4320981204509735, 1.1913315057754517, -0.413136750459671, -0.5797063112258911, -0.14392876625061035, -0.2825395464897156, -0.5993536114692688, -0.04819845035672188, 1.9451429843902588, -0.28547403216362, 0.928412139415741, 1.2813236713409424, -1.7503067255020142, 0.06126120686531067, -0.6697698831558228, -1.0559138059616089, -0.08090837299823761, 1.9954532384872437, 0.09974882751703262, 0.19062110781669617, 0.92920982837677, -0.07724200934171677, 0.04001360014081001, -0.38488322496414185, -1.7777899503707886, -0.8766881227493286, 0.3561287224292755, -1.3046120405197144, 1.570334553718567, -1.544576644897461, -1.013330101966858, 1.2390426397323608, -0.016317134723067284, 0.4276975691318512, 1.6059880256652832, -0.6601152420043945, -1.3751699924468994, 0.5371461510658264, 0.6328049898147583, 1.3125749826431274, -0.004499268252402544, -0.9667574763298035, -1.1068873405456543, 2.0261435508728027, 0.04777267947793007, 0.9840747714042664, 0.8362537622451782, 0.24535129964351654, 1.0171860456466675, 0.6056149005889893, -2.2656936645507812, 0.2808626890182495, 0.9239086508750916, 0.09772691875696182, -0.5783068537712097, 0.9745100736618042, 1.47055184841156, 0.5235981345176697, -0.1579643338918686, -1.9197087287902832, 1.3827532529830933, -0.21715682744979858, 1.0532565116882324, -0.312197208404541, 0.5557035207748413, -1.315279483795166, 1.4617539644241333, 1.4210023880004883, -0.9628853797912598, 0.519599974155426, 1.0629349946975708, 0.10929617285728455, 0.3772501051425934, -0.7805466651916504, -0.8410292863845825, -1.9581186771392822, 2.0550663471221924, 1.0999079942703247, -0.22821854054927826, -0.5063814520835876, 0.24353821575641632, 0.4788481593132019, 0.768149733543396, 2.1877970695495605, 0.6934813261032104, -0.8354253768920898, -0.44464972615242004, -0.39570966362953186, 1.1168808937072754, 0.8514163494110107, 0.26767951250076294, -1.048231840133667, -0.0028839074075222015, 0.7379351854324341, 0.33359861373901367, 0.017503289505839348, -1.9115248918533325, -0.5315734148025513, 0.16861209273338318, -0.3122827112674713, -0.33844050765037537, -0.7671586871147156, 0.11329708993434906, 2.1465768814086914, -1.0203604698181152, -1.522189736366272, -1.7810218334197998, 1.793810248374939, -1.3306598663330078, 0.5244878530502319, -1.0737390518188477, 0.4687357544898987, 0.8411337733268738, -1.214410662651062, -0.3533278703689575, 0.23563259840011597, -0.674135684967041, 1.7231377363204956, 0.9266606569290161, 0.8744996190071106, 0.49777141213417053, 0.766638994216919, 0.89459228515625, 0.1889183670282364, 1.6620959043502808, 1.4799937009811401, 0.03967577591538429, -1.2369825839996338, -0.8049063086509705, 2.0650296211242676, 0.010858208872377872, -0.8202160000801086, -0.781944751739502, 0.18500244617462158, 0.5510308146476746, -0.3319495618343353, 0.49421605467796326, 2.1201586723327637, 0.5703182816505432, -1.862170696258545, 0.8622077107429504, 0.5813744068145752, 0.44028761982917786, -0.9158450961112976, -1.4194693565368652, -0.8003618717193604, 1.1785002946853638, 0.06512036919593811, 0.7658169269561768, 0.21465440094470978, 0.4827728867530823, 0.5195794701576233, 1.6805731058120728, 2.069927930831909, -0.35185930132865906, -1.1101998090744019, 0.5222009420394897, -1.1473103761672974, 0.03677859902381897, -1.543652057647705, -0.4846216142177582, -1.4053901433944702, -0.7590053677558899, 0.672885537147522, -0.46553051471710205, -1.8308398723602295, -1.0262632369995117, 0.8064882159233093, -0.40949442982673645, 1.3457530736923218, 0.7172008752822876, 0.5007305145263672, -1.7688581943511963, -0.8535572290420532, 1.3572683334350586, -0.4874793589115143, -1.4270308017730713, 1.7417795658111572, 1.7147184610366821, -0.6444162130355835, -0.2994004189968109, 1.2980960607528687, -0.0015494036488234997, -0.6137507557868958, 0.22955113649368286, 0.10614778846502304, 0.7468500137329102, -0.10358455777168274, -0.4594631791114807, -0.7603569626808167, -1.4620436429977417, -0.076580710709095, -0.906152069568634, -0.5490915775299072, 0.12505148351192474, 2.209169387817383, -0.7523554563522339, 1.717003345489502, -0.364829957485199, -0.015819858759641647, 0.2813164293766022, -0.48601004481315613, -0.877278745174408, -0.8783281445503235, 0.09685397893190384, 1.783456802368164, -0.044760555028915405, 0.2149365395307541, 2.7882559299468994, 0.13011832535266876, 1.106141209602356, 0.023500699549913406, -0.014720028266310692, -1.2015340328216553, 1.2851758003234863, -0.8974612951278687, -0.9625680446624756, 1.0667330026626587, 0.5818657279014587, 0.35095641016960144, -0.0596785694360733, 0.4491604268550873, -1.5406755208969116, 0.4582699239253998, 0.28797388076782227, -1.5320625305175781, 0.07167594879865646, 0.002898570615798235, 0.5335018634796143, 1.8857815265655518, -0.6382583379745483, 1.9088647365570068, 0.7718583345413208, -0.9390057921409607, 0.6841213703155518, 1.9617671966552734, 0.7530504465103149, -0.20557454228401184, -1.1669156551361084, -0.3835839331150055, -0.41360992193222046, 0.54840087890625, -0.809014618396759, 0.4443081021308899, -0.37727370858192444, 0.03540093079209328, -0.5865667462348938, -0.4903233051300049, 0.38808485865592957, 0.1536247432231903, -1.2423642873764038, 0.05754091963171959, 0.6516455411911011, -0.49132755398750305, -0.2501773238182068, -2.0166923999786377], [-0.42076876759529114, 0.8722673654556274, 0.20580266416072845, -0.8376681208610535, -0.17896665632724762, -0.2066899538040161, -0.10964188724756241, 0.47340330481529236, 0.268426775932312, -0.18529553711414337, -0.32838577032089233, 0.839391827583313, 0.27176982164382935, -0.6562963128089905, -0.3407156765460968, 0.3201647400856018, 0.08940169960260391, -2.108193874359131, 0.06066741421818733, -0.695096492767334, -0.6886349320411682, 0.6484248042106628, 0.9546211361885071, 0.08022969216108322, 0.7390885353088379, 0.8353775143623352, -0.4372696578502655, -0.19480131566524506, 2.0425589084625244, -0.8633221983909607, -1.825386881828308, 0.6039763689041138, -0.4626877009868622, -0.6211137175559998, -0.7850330471992493, -0.9621818661689758, 0.15422791242599487, -0.07887844741344452, -1.3170151710510254, -0.8217487931251526, 1.3817965984344482, -0.19656796753406525, -1.9188560247421265, 0.10090683400630951, 0.714224100112915, 0.09513072669506073, 0.8140015006065369, -1.2066394090652466, -0.0737561285495758, 1.2079674005508423, -0.2636440396308899, -0.12535613775253296, 1.6695327758789062, -0.3405311107635498, -1.4651139974594116, -0.7987739443778992, -0.5399860143661499, -0.7830238342285156, 0.16478228569030762, -0.7253025770187378, 0.1303965300321579, -0.34548553824424744, 0.5794007182121277, 0.5374912619590759, 0.6667871475219727, 1.2436341047286987, -0.3596942126750946, 1.9512863159179688, -0.1648784577846527, -0.3058415651321411, 0.5112258791923523, -0.2736671268939972, -2.21407413482666, 1.6463361978530884, 0.1528620421886444, 0.4011864960193634, -0.6686781644821167, 1.4320056438446045, 2.0044949054718018, -0.45333224534988403, 1.628203272819519, -1.3132646083831787, 0.07072784006595612, 2.1643292903900146, -0.046906862407922745, 0.5313138365745544, -0.24582690000534058, 1.3299189805984497, -0.3855336308479309, -1.2753424644470215, 0.3286348879337311, -1.0010796785354614, 1.0034832954406738, 1.9590691328048706, 0.18814684450626373, -1.025420904159546, -1.6414910554885864, -2.6651179790496826, 0.1851874738931656, 0.3353953957557678, -1.1031861305236816, 1.4053092002868652, -0.24039195477962494, 1.3106715679168701, -1.1463146209716797, -0.5692337155342102, -1.6543926000595093, -1.9035563468933105, 1.466591238975525, -0.18066665530204773, -0.5344150066375732, -0.35284826159477234, -3.567379951477051, 0.37953540682792664, -0.44360193610191345, -0.6536209583282471, -0.30090034008026123, -1.26509428024292, -1.5125013589859009, 0.2570762038230896, -0.9022038578987122, 0.11170197278261185, 1.1169718503952026, 3.153005599975586, -0.06656183302402496, 1.5215363502502441, 0.011322357691824436, 0.19338944554328918, 0.4386520981788635, -0.8857473731040955, -0.48867785930633545, -1.3296480178833008, 0.5415469408035278, 0.17242999374866486, 0.4673348367214203, 0.6285696625709534, -0.10554757714271545, -0.644997775554657, -0.24508662521839142, 0.08481323719024658, 1.2910456657409668, -0.8671441078186035, 1.9706202745437622, 0.572091817855835, 1.8396629095077515, 0.520095705986023, 0.6787846684455872, -0.6922969818115234, -0.8359330296516418, 0.2604497969150543, -0.3994333744049072, -0.2892944812774658, -0.37499022483825684, 0.4971042573451996, 0.8667978048324585, 0.7621205449104309, -1.1804484128952026, 2.1806163787841797, 0.05560356006026268, 1.957778811454773, 0.7173798084259033, 0.018838893622159958, 0.509933352470398, 0.253003865480423, 0.233873650431633, -0.13492901623249054, -2.5136940479278564, 0.6064164638519287, 0.0063196332193911076, -0.1754126250743866, -0.07506626844406128, 1.1965792179107666, 1.157832384109497, 0.33706048130989075, -0.5067527890205383, 2.747856378555298, 0.8218541145324707, -1.3178702592849731, 0.20281222462654114, 0.7033458352088928, -0.6275786757469177, -2.183793067932129, 0.39165329933166504, 0.6400743722915649, 0.20685511827468872, -0.44033709168434143, -0.2886705696582794, 1.7417644262313843, 2.1659929752349854, 1.8403396606445312, -0.2789444327354431, -0.1585930436849594, -0.9091548919677734, -0.10281740128993988, -0.49238288402557373, 0.8693049550056458, 0.7544078230857849, -0.9714236855506897, 0.9606783986091614, -0.7689658999443054, 0.5638086795806885, 0.4210876226425171, 0.42791950702667236, 1.4419629573822021, 1.580372929573059, 0.3995200991630554, 0.014906044118106365, -0.36732473969459534, -0.04744506999850273, -0.3960524797439575, -0.8790180683135986, 0.5775606632232666, -1.3597137928009033, -0.3204246461391449, 0.6688392758369446, -0.04946834594011307, -0.4262748956680298, -0.3668794333934784, 0.7189915776252747, -0.450630784034729, -1.7888942956924438, 0.3306223452091217, -0.05848316475749016, 0.37571340799331665, -0.49681276082992554, 0.5391559600830078, 0.319355309009552, 0.09380915760993958, 0.47107642889022827, 1.083071231842041, -0.16402332484722137, -0.04872468113899231, -0.1530151218175888, -1.3294447660446167, 2.039095640182495, -0.486301064491272, 2.222320318222046, -0.6819795370101929, -1.8274351358413696, 0.16489030420780182, -0.7918894290924072, -1.2071466445922852, 1.0015532970428467, -0.7199354767799377, 0.08771281689405441, -1.1653614044189453, 0.2628721594810486, 0.7789037227630615, -1.1624611616134644, -1.2055472135543823, -0.9221780896186829, -0.6643026471138, 0.5223342180252075, 0.7767035365104675, 0.9551576375961304, 0.8989344239234924, -0.22828218340873718, 2.758500099182129, -0.1968706250190735, -0.3130638003349304, 0.1158086284995079, 0.15684792399406433, 0.06679907441139221, 0.21185798943042755, -0.15846507251262665, 0.6416901350021362, -0.4553528428077698, 0.5392190217971802, 1.2693101167678833, -0.2969124913215637, -0.6458972692489624, 0.6074292063713074, 1.444886326789856, -1.5974891185760498, 0.5830920338630676, -0.42604300379753113, 0.5749384164810181, 0.8085282444953918, -0.16844482719898224, -0.39118459820747375, -2.6642723083496094, -0.23323145508766174, -0.3449273705482483, -0.4909139573574066, 0.12134251743555069, -0.9663845896720886, 0.5154266357421875, 1.1769801378250122, 1.2711998224258423, 1.1664042472839355, -0.5855638384819031, 1.8246320486068726, 1.4336206912994385, 1.2483303546905518, -0.6139915585517883, -0.29744768142700195, -0.2906346321105957, -2.380328893661499, 0.47803422808647156, -0.6781701445579529, 0.25284239649772644, -0.36029309034347534, 0.20536395907402039, 1.354303002357483, -1.0671483278274536, -0.8093540668487549, 0.26994195580482483, -1.5180414915084839, 1.1027557849884033, 0.0029011673759669065, 1.9831299781799316, 0.6847614049911499, 0.5177114009857178, -0.5261871218681335, -1.217687964439392, 0.9229647517204285, -0.31366172432899475, -1.192770004272461, 0.15225957334041595, 0.7934600114822388, -1.1504420042037964, -2.014185667037964, 2.211308717727661, -1.325778603553772, 0.450838178396225, 0.5618963241577148, -0.13910259306430817, 0.3926965296268463, -0.10396166890859604, -1.7875590324401855, 1.0140877962112427, 0.48652684688568115, -0.15795378386974335, -0.3637785315513611, 0.9975647330284119, -0.8851308822631836, -0.48888471722602844, -0.535876452922821, -1.726196527481079, 1.5611859560012817, -1.0561984777450562, 0.09568905830383301, -0.6391360759735107, 0.6452407240867615, 0.29919692873954773, 0.8261969685554504, 0.09933248907327652, 0.6912670731544495, 0.09064076095819473, -0.6411095857620239, -0.1014857292175293, -0.9557371139526367, 0.36492297053337097, 0.9106841087341309, -0.27071136236190796, -1.502861499786377, -1.5828354358673096, 0.5679492354393005, -0.9649971127510071, 0.1024821475148201, 2.0067880153656006, 1.1888742446899414, 0.44819214940071106, 1.1248795986175537, -0.7042650580406189, -0.1635291427373886, -1.2702792882919312, 0.16127046942710876, 0.16302624344825745, -0.5149276852607727, 1.0897520780563354, 2.333514928817749, -0.8602740168571472, -0.2713530659675598, -0.8915319442749023, -0.3675551116466522, 0.5001800060272217, 0.4209997355937958, -0.5461915731430054, 0.941231369972229, 0.11386159807443619, 0.13942360877990723, 1.1945993900299072, 0.7845498919487, 0.2943160831928253, 0.4585892856121063, -0.45962321758270264, 1.1588925123214722, 1.184722900390625, -0.5143018364906311, -0.2695208787918091, -0.17401240766048431, 0.6232379078865051, -0.6786162257194519, -0.8830277323722839, 0.42650556564331055, 0.6481844782829285, -0.7293606996536255, 0.33163049817085266, -1.0813817977905273, 0.07408183068037033, 1.281235694885254, 1.5193543434143066, -0.648891270160675, 2.0526602268218994, 0.10125252604484558, -0.1035248190164566, 0.2320033609867096, -0.713038980960846, 2.2511088848114014, 0.7013658881187439, -1.146642804145813, -2.4390218257904053, -0.32281699776649475, -0.2603926360607147, 0.5264928340911865, -1.32516348361969, -0.4250880777835846, 0.15700337290763855, -0.33982616662979126, 0.7728550434112549, 0.4989268183708191, 0.19201652705669403, -0.49514859914779663, -1.5023462772369385, -0.0678960531949997, -0.45207059383392334, 0.18488822877407074, 1.3706638813018799, -0.3512956202030182, 0.03774770721793175, -2.682858943939209, -2.2051119804382324, 1.7024813890457153, -0.896790623664856, -1.3540762662887573, 0.10457906872034073, 0.7837032675743103, 0.12545402348041534, -2.0294153690338135, 1.2957584857940674, 0.5433436632156372, -1.3726774454116821, 0.15341028571128845, -1.9427437782287598, -0.7271081209182739, 0.46401840448379517, -1.5807116031646729, 0.06933827698230743, 0.015403399243950844, -2.0141286849975586, 1.0797663927078247, 0.27775365114212036, -0.037425823509693146, 0.974332869052887, -1.1123408079147339, 0.46836891770362854, 0.8778805732727051, 1.2473325729370117, 0.5840162634849548, 0.8525116443634033, 0.5680970549583435, -0.4572485089302063, 1.5068405866622925, 0.16357018053531647, 1.6944730281829834, -0.07229459285736084, 1.6773669719696045, 0.7020131945610046, 0.291059672832489, 1.0767312049865723, -1.2507250308990479, 0.3235536813735962, -0.8743782043457031, 0.3218063712120056, 0.6646718382835388, -0.2834744155406952, 0.701153576374054, 0.8036859035491943, -0.5873138308525085, -0.28883498907089233, 1.009925127029419, 0.06997399777173996, 0.1955799162387848, 0.6965200304985046, 1.5603318214416504, -1.698055624961853, 0.6614185571670532, 1.220550537109375, -2.3489599227905273, 0.9120873808860779, 0.48555952310562134, 1.6010501384735107, -0.6952074766159058, -0.20169082283973694, 0.3433513641357422, -0.7505568265914917, -0.588672399520874, 0.44633129239082336, -0.44492867588996887, -0.23822486400604248, 0.00015731347957625985, -1.5555473566055298, -0.6039047837257385, -1.5680361986160278, 0.437923401594162, -0.6509178280830383, -1.484330654144287, -0.9151312708854675, -0.4365929067134857, 0.2844284176826477, -0.14480875432491302, -0.3686006963253021, 0.06364942342042923, 1.0364489555358887, -0.010534249246120453, 0.21413050591945648, -0.3877938687801361, 0.7896130681037903, -1.0495786666870117, 0.05704221874475479, 0.7788371443748474, 0.3398415446281433, 0.6290876865386963, -1.3024687767028809, 2.2575626373291016, 0.03661039099097252, -1.257412075996399, 0.5027227997779846, -0.19434750080108643, 1.5317267179489136, -1.3098665475845337, 1.3142528533935547, 0.2564638555049896, 0.7074726819992065, -0.45874205231666565, -1.5407803058624268, -1.4537376165390015, -1.4896796941757202, 0.6200991868972778, 1.1333240270614624, -0.930895209312439, 0.865662693977356, 1.463139533996582, 2.463893175125122, -1.4358437061309814, 0.6851364970207214, 1.6366004943847656, 0.4640394151210785, -1.0441814661026, 1.128442406654358, 0.6790325045585632, 0.4798799455165863, 0.32916927337646484, 0.4238390028476715, -0.4028279483318329, 0.04406903684139252, 0.6291986107826233, -0.31699854135513306, 0.8214695453643799, -1.138755202293396, -0.9969011545181274, 0.42314663529396057, 0.3804468810558319, -0.8745928406715393, -1.1642893552780151, -0.3325745165348053, -1.345076322555542, -2.3127362728118896, -0.31432321667671204, 0.7001954317092896, -0.02508622407913208, 1.404043197631836, -0.531739354133606, -0.9105138182640076, 0.5577948689460754, -0.00704942736774683, -0.18336884677410126, -0.7982362508773804, -1.0980184078216553, 0.360307514667511, -1.3334732055664062, 0.22561374306678772, -0.04933594912290573, 0.27462470531463623, 0.7189092040061951, 3.086655855178833, 1.0473783016204834, 0.6347599029541016, 0.49760305881500244, -0.239753857254982, 1.0140882730484009, 0.46229878067970276, 1.1451281309127808, -0.6057462096214294, -1.0092189311981201, 0.9305069446563721, 0.31696128845214844, -1.2873215675354004, 0.5989336967468262, -0.09807451069355011, -0.767374575138092, -0.5549148321151733, -0.6439650058746338, -0.7006233334541321, 0.9856468439102173, 0.478305459022522, 0.7868474721908569, 1.944644808769226, 1.0575931072235107, 0.4860955476760864, 0.1664057821035385, 0.06886659562587738, -0.25343990325927734, -2.152822971343994, 1.5827990770339966, -0.6552898287773132, 0.33805468678474426, -0.5389668345451355, 0.12796823680400848, 0.8414168953895569, 0.6656104326248169, -1.1023973226547241, 0.25591740012168884, 2.4765262603759766, -0.3257102072238922, 1.4255201816558838, 0.7873745560646057, -0.4575423002243042, 0.26873254776000977, 0.2185892015695572, 0.1115892231464386, 1.8317404985427856, 0.6093370318412781, -0.8385862112045288, -0.2051897794008255, -0.6937182545661926, -1.3153587579727173, -0.6824019551277161, 1.9269094467163086, 0.7520137429237366, -0.05207158997654915, 0.011605633422732353, -0.13797371089458466, -2.4117236137390137, -2.2036654949188232, -0.35018354654312134, 1.4109702110290527, 0.5177635550498962, -0.2875041365623474, -1.275762677192688, 0.14554379880428314, -0.9023498296737671, -0.45839163661003113, -1.7203480005264282, -0.8335374593734741, -0.4888819456100464, 0.8083395957946777, -1.5173351764678955, -1.0517247915267944, 0.6381778717041016, 1.5106273889541626, 0.8986808657646179, 1.2323123216629028, -0.3282332718372345, -0.2568208873271942, -0.930720329284668, 1.0267689228057861, 0.8131886124610901, -0.5776023864746094, -1.2469186782836914, -0.894873857498169, -0.1448650062084198, 0.0936775729060173, -0.13600480556488037, -0.6472762823104858, 1.3357815742492676, 0.5892838835716248, 0.18239091336727142, 0.6899415850639343, 0.19213995337486267, -0.16081735491752625, -0.8096962571144104, -0.7538608312606812, 0.600136935710907, 0.9572904109954834, -1.386184573173523, 0.31193622946739197, 0.6007880568504333, 0.4926084876060486, 0.34794119000434875, 0.37497538328170776, 0.5750135779380798, -0.6397222876548767, -1.4955543279647827, -0.03317685425281525, 1.2562097311019897, 1.419299602508545, 1.3857665061950684, -0.04834391921758652, 0.620495617389679, 0.9879677295684814, -0.14968085289001465, 1.299999475479126, -0.8057718873023987, -0.2055007815361023, 1.49985671043396, 0.16720178723335266, 0.5297256708145142, 0.39349573850631714, 1.4211223125457764, -1.7636278867721558, 0.5841091871261597, 1.0427656173706055, 1.2904765605926514, -0.9263601303100586, 0.18777644634246826, -0.2776767611503601, 1.1509674787521362, -1.3297027349472046, 0.41629964113235474, 0.1791168600320816, 0.2851170003414154, 0.9655874371528625, 0.23230719566345215, 0.3872886300086975, 1.1162632703781128, 0.5194202065467834, 0.1307007223367691, 0.2288854420185089, -2.6210663318634033, 0.8621183633804321, 0.7670243978500366, -1.5072855949401855, -0.708042562007904, 0.7520782947540283, 1.1109979152679443, 1.553695797920227, -0.9511217474937439, 0.667146623134613, -0.15617771446704865, 0.22713135182857513, -0.2883935570716858, 0.342403769493103, 0.034473225474357605, -0.3512135148048401, 0.5177977681159973, -0.8997806906700134, 0.6065080165863037, 0.35451361536979675, -0.24841028451919556, -1.6050851345062256, -0.429053395986557, -0.5634058713912964, -1.1739227771759033, -0.24661798775196075, -2.375190258026123, -1.784981608390808, -0.982798159122467, 1.3429127931594849, 0.8955475091934204, -0.7464950084686279, 1.2651442289352417, 0.9765414595603943, -1.7903512716293335, -1.3112354278564453, -0.47032779455184937, 0.35464999079704285, 1.413547396659851, -0.24254350364208221, -1.1070128679275513, -1.7450119256973267, -0.7577009797096252, 1.1881146430969238, -0.007223823573440313, 0.36394384503364563, 0.18507593870162964, 0.9159011840820312, 0.7025960683822632, 0.47074246406555176, -0.5220731496810913, 1.760515809059143, -0.39707574248313904, 0.38205549120903015, -0.3412896990776062, 0.7108174562454224, 0.812171459197998, 0.6629102230072021, 0.023581046611070633, 0.664107084274292, 0.6096963882446289, 0.2415751963853836, 0.6813554167747498, 0.7617034316062927, -0.6048823595046997, -0.0878811627626419, -0.3585827946662903, 0.15297259390354156, -0.5312278866767883, -0.5244662761688232, -0.34163743257522583, 0.26993921399116516, 0.8086004853248596, -1.0565983057022095, -1.0441535711288452, -0.4230259656906128, 1.9517265558242798, -0.8694515824317932, 1.4005721807479858, -1.435996651649475, 0.9091400504112244, -0.2814103066921234, -0.3924921452999115, 1.4494516849517822, -0.060358837246894836, -1.420397162437439, 0.1369522660970688, -0.13876356184482574, 0.4879645109176636, -1.6145912408828735, 0.09526552259922028, 0.4620353877544403, -1.4857597351074219, 0.6998092532157898, -0.5125104188919067, 0.12101034075021744, -0.47123637795448303, 0.6208529472351074, 1.9577538967132568, 0.9307848811149597, 0.38266998529434204, 0.20038846135139465, -0.10977964103221893, 0.7168314456939697, -0.7696488499641418, 0.4801716208457947, 0.029070734977722168, 0.3132827877998352, 0.23383729159832, 0.05068575218319893, 0.05536798760294914, -0.4731769561767578, -0.4088355004787445, 0.6017646193504333, -1.8688150644302368, 0.820259690284729, -0.1830800175666809, 0.7532258629798889, 1.0711592435836792, 0.9401084780693054, -0.8631116151809692, -0.4581993818283081, 0.016100525856018066, 0.8618762493133545, -1.6418626308441162, 1.0070545673370361, 1.8967270851135254, -0.788529634475708, -2.3639636039733887, -0.8352887034416199, 0.20178155601024628, -0.4806755483150482, -1.453299641609192, -1.1108295917510986, 0.017753371968865395, -0.10489669442176819, -0.025999464094638824, 2.0453226566314697, 0.8630479574203491, -1.1707104444503784, 1.0269256830215454, 0.46285006403923035, 0.49017298221588135, -1.52603280544281, 1.2583129405975342, -1.5258933305740356, -0.4621827304363251, 1.101668119430542, 0.6798592209815979, -0.8984135985374451, 0.15818791091442108, -0.1487397998571396, -0.1446990817785263, 0.3606647253036499, 1.3111470937728882, -0.4950231611728668, 2.180603504180908, -0.1075749322772026, -0.6258134245872498, 0.6040247678756714, 0.8877120018005371, -1.5470530986785889, 0.18288330733776093, 0.20788273215293884, -0.7083752751350403, 1.561141014099121, -0.21588720381259918, -0.32329311966896057, 0.034182190895080566, -0.09538881480693817, -1.9473670721054077, 0.616409957408905, -1.316601037979126, -0.15893003344535828, 1.5744144916534424, -0.32167166471481323, 0.9801638722419739, 0.5470630526542664, 1.1898889541625977, -0.7752795815467834, 0.5109574794769287, 0.9674830436706543, 0.24435068666934967, 0.1316039115190506, 1.0840904712677002, -0.42440202832221985, -0.25543949007987976, 0.25981876254081726, -1.7529270648956299, -1.5521454811096191, -0.40057021379470825, 1.4109586477279663, -0.1185697689652443, 0.6134507060050964, -0.25276750326156616, -0.9683640599250793, 0.37138110399246216, -1.0184259414672852, -0.9644817113876343, 0.30516788363456726, 1.2987537384033203, -1.3604763746261597, -0.5023302435874939, 1.0389140844345093, -1.510963797569275, -0.18530182540416718, 1.1522654294967651, 0.01812741905450821, 1.5229099988937378, -1.1033508777618408, 0.08919890969991684, -2.0781266689300537, -1.5266164541244507, 0.20882846415042877, -0.7670835256576538, 0.8237601518630981, -0.08240895718336105, 1.145738959312439, 0.8094791769981384, 0.5239651799201965, 1.6119617223739624, 1.1321897506713867, 1.7002193927764893, -1.6137086153030396, 2.0832669734954834, -0.23530980944633484, 0.23396991193294525, -0.18408378958702087, 0.7223598957061768, 1.1485010385513306, -1.0777766704559326, 1.2525850534439087, -0.2740914821624756, 0.12026648223400116, -0.6220758557319641, 0.12266437709331512, 0.4161895513534546, -0.7043570280075073, 0.3150365650653839, -0.1544557511806488, -0.4294130206108093, 0.6886540055274963, 0.2890981435775757, 1.5470975637435913, 0.5930388569831848, -0.7476702928543091, 0.09527233242988586, 0.7798019051551819, 1.3674402236938477, -0.8044649362564087, 1.0422723293304443, -1.4769368171691895, -0.2679428458213806, 0.20818638801574707, 0.3263913094997406, 0.2598503530025482, 0.3821341097354889, -0.6543785333633423, 0.9718729257583618, -1.0729808807373047, -0.7023650407791138, -1.589735746383667, 0.16969658434391022, -1.1783664226531982, -1.656416416168213, 0.2948993146419525, 0.8908298015594482, -0.24290141463279724, -0.9724398255348206, -0.469098299741745, -0.5031152367591858, 0.66239333152771, 0.8189464211463928, -0.2335730493068695, -0.8367612957954407, 0.5494939088821411, -1.4966485500335693, -0.200717493891716, 0.8158567547798157, -1.5278775691986084, -0.3642926812171936, 0.6280428767204285, 0.7618491053581238, 0.44444072246551514, 0.6031437516212463, -0.3518674373626709, 1.176242709159851, -0.11485552787780762, -1.3249695301055908, 1.94864821434021, 0.37098804116249084, -0.7037099003791809, -2.137820243835449, -0.467644602060318], [0.8603707551956177, -0.8290388584136963, 0.07806616276502609, -0.4838014245033264, -0.385576993227005, 1.3833365440368652, 0.2363838106393814, -0.761760413646698, -1.3488001823425293, 1.5065383911132812, 0.00760627631098032, -0.43555575609207153, -0.7060657143592834, 0.935523271560669, 0.4696507155895233, 1.088220477104187, 0.055442553013563156, 2.1473047733306885, -1.5022237300872803, 0.3281601369380951, -0.10435440391302109, -0.5424889326095581, 0.18814770877361298, -0.32879722118377686, 1.0990322828292847, -0.10413683205842972, 0.2334049493074417, -0.00011090753105236217, -0.2800007164478302, -0.8416448831558228, -0.9658145904541016, 0.9243929982185364, -0.8769184947013855, -0.17014528810977936, 0.11648009717464447, 0.2829107344150543, 0.5306826233863831, 0.419878214597702, 1.346666932106018, 2.002558708190918, 1.3364927768707275, -0.05252915620803833, -0.19096529483795166, -0.21985961496829987, 0.7321597337722778, -1.6009660959243774, -1.1009407043457031, -1.675158977508545, 0.29983142018318176, -0.5017136335372925, 1.6451433897018433, 1.4167100191116333, 0.10988537222146988, -0.7972184419631958, -0.34145885705947876, -2.4833431243896484, 0.6300235986709595, 1.7181142568588257, 0.6163660883903503, 0.9490844011306763, -0.01433589681982994, -0.06355258822441101, 0.4850770831108093, 0.6910810470581055, 0.6345100998878479, -0.35732588171958923, -0.06999161094427109, 0.5938746929168701, 0.7346901297569275, 0.2773742377758026, 0.8550630211830139, -0.7154022455215454, -0.03269267454743385, -0.6559717655181885, -0.310346394777298, 0.061127908527851105, -0.15999534726142883, -0.39707303047180176, -1.2878260612487793, 0.6326044797897339, -1.1371196508407593, 0.2798506021499634, -0.42664840817451477, -1.9432930946350098, -0.6649377346038818, -0.9764291644096375, 0.018128976225852966, -0.2823021411895752, 1.3060212135314941, -0.09722678363323212, -0.9126884341239929, 0.006248808931559324, -0.14134226739406586, -0.43986019492149353, -0.11895215511322021, 0.683051586151123, -1.2435569763183594, -0.3685092329978943, -0.13053439557552338, 1.043651819229126, 0.1908150166273117, -0.5777156352996826, 0.14472247660160065, -1.8578438758850098, 0.9468550086021423, -0.42594876885414124, 0.13562321662902832, 0.6330200433731079, 0.11280753463506699, -0.5314456224441528, 0.40445834398269653, 2.0696771144866943, -0.6311838626861572, -0.9770398139953613, 0.30817711353302, -1.4693127870559692, -0.027701612561941147, 0.4845960736274719, -0.12409088015556335, -0.49940624833106995, -0.21527881920337677, -0.2532093822956085, 2.8667078018188477, -1.738273024559021, 0.8159451484680176, 0.13955022394657135, 0.44322696328163147, 0.6185337901115417, -0.31503772735595703, 0.3656032681465149, -0.6101248860359192, 1.5270310640335083, -0.015686513856053352, -0.9761942625045776, 0.38267943263053894, -0.5941638350486755, 0.40260910987854004, 0.6593822240829468, 0.12136095762252808, -0.905073881149292, -1.808498740196228, -0.4300841987133026, -0.6871418952941895, -0.6886601448059082, -0.884956955909729, 1.6169782876968384, -0.7502323389053345, -2.0243868827819824, -0.025459744036197662, 0.9267849922180176, -1.7511403560638428, -2.1482152938842773, 1.3391624689102173, -0.0221993550658226, 0.5987427234649658, 1.0513900518417358, 1.4085218906402588, 0.6893993616104126, 0.367463618516922, 1.3588128089904785, -1.305794596672058, 1.1023342609405518, -0.8308084607124329, 0.24830377101898193, -0.15928475558757782, -0.4856649935245514, -1.692833662033081, -1.0749781131744385, 0.32394200563430786, 0.7718623280525208, 1.2219165563583374, 0.5603007078170776, -0.37934407591819763, -1.462188720703125, -0.9718120098114014, -1.9762310981750488, -1.5305098295211792, -0.725883424282074, 0.09167879819869995, 0.6653514504432678, 1.4279930591583252, 0.8294666409492493, -0.4100862443447113, 0.03334929049015045, -0.18349821865558624, -1.2198681831359863, 0.515953540802002, 0.5808606147766113, -0.6091555953025818, 0.03123876266181469, -1.1825882196426392, -0.1680850088596344, -0.37282130122184753, -0.6273922324180603, 0.512538492679596, -0.848116397857666, 0.629683256149292, -0.9958512187004089, -0.7076427936553955, 0.9553712010383606, -0.9322553277015686, 0.11519728600978851, 0.7347676753997803, -0.6249017715454102, 1.4381612539291382, 1.1498583555221558, -0.44212421774864197, -0.9844909906387329, 0.31625378131866455, -0.18573643267154694, 0.421234130859375, -1.7943072319030762, 0.47423461079597473, -0.7898942232131958, 1.125004529953003, 0.1633022129535675, -0.25564494729042053, 1.2067654132843018, -0.42559266090393066, -0.25340282917022705, -0.8698945045471191, 0.18477845191955566, 0.1326543092727661, -1.5806413888931274, 0.3668227195739746, -0.7564789056777954, -1.2412408590316772, 0.9731586575508118, -0.9769534468650818, -0.5411693453788757, 0.33729639649391174, -0.25631484389305115, 1.594041347503662, -1.3359375, 1.2434632778167725, -1.0174310207366943, -0.49381089210510254, -0.6298227906227112, -0.8081575632095337, 0.7959493398666382, 0.9157412052154541, 0.0742676630616188, -0.844128429889679, 1.1858599185943604, -0.01700161211192608, -1.336842656135559, 0.7601171135902405, 1.2499876022338867, 0.7393130660057068, 0.7050456404685974, -1.251757264137268, 0.18512853980064392, 0.8248860836029053, 2.231510639190674, 0.10422226786613464, -0.16989848017692566, 2.1229100227355957, -0.8216493129730225, -1.2130827903747559, 0.994844913482666, -0.33643364906311035, 0.766558825969696, -0.15377193689346313, -0.7555452585220337, 0.6748928427696228, 0.9719523787498474, 0.375672847032547, -0.6702072620391846, -1.3167612552642822, -0.6188055276870728, -0.26390933990478516, 0.7971177697181702, -0.5549248456954956, 0.039293382316827774, 0.19942472875118256, -1.080772876739502, -0.16213487088680267, 0.5462612509727478, 1.5999550819396973, -0.2735287845134735, -0.9897031784057617, 0.6955113410949707, 1.4573498964309692, -1.0657198429107666, -0.7843790054321289, 0.5619887113571167, 0.48925161361694336, -0.5316212177276611, 0.07523950189352036, -0.38346150517463684, -0.5831266641616821, 1.0136646032333374, -1.1553890705108643, -0.23127859830856323, 0.30565884709358215, 0.41739901900291443, 0.18730227649211884, -0.6686938405036926, 0.15497197210788727, -1.4023181200027466, 0.9839892387390137, 0.4931151568889618, -1.2409576177597046, 0.6403005123138428, 0.9293792247772217, -0.01799401454627514, 1.6945785284042358, 0.43727216124534607, -0.39019522070884705, -0.997191846370697, 0.8135731816291809, -1.9387882947921753, -0.9376932978630066, -1.7734813690185547, -0.9850464463233948, -0.4241761565208435, -1.6396135091781616, -1.5315674543380737, 0.7993326187133789, 1.542923927307129, 1.1174405813217163, -0.6095865368843079, -1.0576372146606445, 0.7723296880722046, 0.6437529921531677, 0.4152334928512573, 1.0892159938812256, -1.5980654954910278, 1.1143572330474854, -1.3924694061279297, 0.04891206696629524, -0.2845301330089569, 0.36122414469718933, 1.083783745765686, 0.42449185252189636, 1.7514296770095825, -1.1116142272949219, 0.5258364677429199, -0.29203248023986816, 0.6022181510925293, -0.6358943581581116, -0.1957067847251892, 0.9664546251296997, -0.9609912633895874, 0.7885343432426453, -0.3658393621444702, 0.47443726658821106, -0.2958965599536896, 1.3211621046066284, 0.6990079283714294, -0.8928418159484863, -0.44806107878685, 1.0082942247390747, 0.26830026507377625, -0.368839830160141, 1.0295315980911255, -0.2641652226448059, -0.051510050892829895, -1.4946932792663574, -0.6601313948631287, -0.7051141262054443, 1.7619614601135254, -0.48719319701194763, -0.3635334074497223, -0.857160210609436, 1.6072543859481812, -1.3145310878753662, -1.9688937664031982, 0.24480582773685455, -0.673862099647522, 0.1557191014289856, 0.37009793519973755, -1.006437063217163, 1.7320277690887451, 0.2934814989566803, 0.10270630568265915, 0.3327682912349701, 1.1167155504226685, -0.5942149758338928, 0.13343936204910278, -0.9992079734802246, -1.032006025314331, 0.09732238948345184, 0.05938514694571495, -0.3335283100605011, -1.9236847162246704, 0.964668869972229, 1.722794771194458, -0.5963695049285889, 1.0727320909500122, 1.4130699634552002, -0.6905460357666016, 2.374464750289917, -2.7001187801361084, 0.8448801636695862, -0.24249392747879028, 0.7656868100166321, 0.3107876777648926, 1.9902312755584717, -0.6615805625915527, -0.33845946192741394, -1.125622034072876, 1.1105281114578247, -0.434325635433197, 0.46106183528900146, -1.4461047649383545, 0.45116639137268066, 2.3525760173797607, -0.9643703103065491, 0.3880435824394226, -0.6172526478767395, -0.8807728290557861, -0.8007717132568359, -0.5952287316322327, 0.7763840556144714, 0.6261849999427795, -0.4372258186340332, -0.5374585390090942, 0.25829997658729553, 0.5093733668327332, -0.9053446650505066, 0.82115238904953, 1.7706940174102783, 0.5437185764312744, 0.7987714409828186, -0.84308260679245, 0.9340627193450928, 0.46181371808052063, -1.2550277709960938, 1.8028556108474731, 0.2649902105331421, 0.2680511772632599, 1.7082823514938354, -0.03933725506067276, 0.7772359848022461, -1.6603001356124878, 0.560267448425293, 0.5169631242752075, 0.41250211000442505, 0.9558891654014587, 0.36386799812316895, 1.0083990097045898, 0.10591526329517365, 0.40341296792030334, 0.5902770757675171, -0.2952054440975189, 0.46929267048835754, 0.5689409971237183, 0.008211902342736721, 1.2671468257904053, -0.15563331544399261, 1.6493453979492188, 1.67616868019104, -0.6898873448371887, -1.1148204803466797, -0.7245863080024719, -0.07613757997751236, -0.4796432852745056, -0.10169114917516708, -0.8321195840835571, 0.8372453451156616, -0.08128543198108673, -0.26079949736595154, -0.8414397239685059, 0.16116319596767426, -0.8901222348213196, -0.6841452121734619, -1.081868052482605, 1.2189589738845825, -0.060568924993276596, 1.5861207246780396, -0.9346954822540283, -1.645614743232727, -0.6643811464309692, 0.27449163794517517, 2.0141866207122803, -1.3512325286865234, 0.311746209859848, -0.7617420554161072, -0.5702797770500183, 0.9054528474807739, -0.17744487524032593, -0.6644821763038635, 0.4484395980834961, -1.112059473991394, -0.9331342577934265, 0.01656854711472988, -0.44565099477767944, 0.4871768057346344, -0.3268841803073883, -0.8220641613006592, 0.972117006778717, 1.3652656078338623, -0.10631027817726135, 1.1324657201766968, 1.542739748954773, 1.6406453847885132, 0.8337463736534119, 1.1073817014694214, 0.5303529500961304, -0.6124736070632935, 0.9943698048591614, 0.10477082431316376, 0.9196898937225342, -0.7782294154167175, 0.16769249737262726, -1.1918416023254395, 0.9077010154724121, 1.8119843006134033, -0.9844454526901245, 1.3540846109390259, 0.49328574538230896, 0.7279882431030273, 1.6013126373291016, 0.4777955412864685, 1.2965030670166016, 0.4483748972415924, -0.9512383341789246, -1.2911983728408813, -0.6379115581512451, -1.1722499132156372, 1.3937326669692993, 0.11544696241617203, 1.3505383729934692, -1.5793046951293945, -0.30337944626808167, 0.787771463394165, -0.19549906253814697, 0.683006763458252, 0.27730169892311096, -0.519097089767456, 0.7998806834220886, 1.0025346279144287, 2.175201892852783, -0.7019662857055664, -0.27862152457237244, -0.23341502249240875, -0.3351476192474365, -1.2691107988357544, -0.5853871703147888, 0.23285837471485138, 0.08193077147006989, -0.8725834488868713, 0.06260227411985397, 0.1024605855345726, -0.8464494347572327, 1.3918730020523071, 0.6943873763084412, 0.5381139516830444, 1.1644808053970337, -2.539202928543091, -0.33165040612220764, 0.4999507665634155, 0.19994907081127167, 0.9880527853965759, -0.19120445847511292, 0.1940106451511383, -0.7963421940803528, 0.3236863613128662, 0.049818482249975204, -0.6455108523368835, -1.781816005706787, 0.18232113122940063, 0.16366177797317505, -0.6743999719619751, 2.5866286754608154, 1.5285717248916626, -0.8242490291595459, -0.9297593235969543, -0.7878058552742004, 0.8027586936950684, -0.8141165971755981, 1.026379108428955, 0.0395674966275692, 1.0701968669891357, -0.6831039786338806, -1.6207512617111206, -0.5653062462806702, 1.7331502437591553, 1.5173307657241821, 0.5512991547584534, 0.19373159110546112, 1.2748894691467285, -1.332194447517395, -0.140138640999794, 0.7017079591751099, -1.1376025676727295, 0.41462820768356323, -1.4903900623321533, -0.4035870432853699, 0.4538792371749878, 0.3617261052131653, -0.36845695972442627, -1.111296534538269, 0.7673032879829407, 0.4496781527996063, 0.7917505502700806, -0.9800673723220825, -0.5213819146156311, -0.2671959400177002, 1.0908540487289429, 1.089490294456482, -0.0479716993868351, -0.5577757358551025, -0.6856155395507812, -0.5653153657913208, 0.08285874873399734, -0.06165109574794769, 0.49777957797050476, 0.2143193930387497, 0.6226942539215088, -1.7649487257003784, 1.068670392036438, -0.5783982276916504, 1.4276992082595825, -0.035843003541231155, 0.7412101626396179, -1.7681667804718018, 0.10526527464389801, 1.003374457359314, 3.0424811840057373, 0.22601290047168732, 1.5129997730255127, -0.5854637622833252, -0.07509937137365341, -0.38331127166748047, 0.4943896532058716, 0.865697979927063, -1.4113905429840088, 0.4869289994239807, 0.6060594916343689, 0.9828478693962097, 0.26576676964759827, -0.8912394046783447, 0.5790532827377319, -1.0997018814086914, 1.666361927986145, 0.826743483543396, -0.7189716696739197, -1.0883597135543823, -1.9540245532989502, -0.12350452691316605, 1.514539122581482, 1.1020746231079102, 0.33152762055397034, -0.1903032511472702, 1.634612798690796, 0.7354193329811096, -1.0601037740707397, 0.9812393188476562, 0.8843576312065125, 0.5837967991828918, 0.2628840208053589, -0.5109888315200806, -1.2889556884765625, 0.5691236257553101, 0.5017961859703064, -0.1143518015742302, 0.9987163543701172, -0.2617933452129364, 0.13522325456142426, 1.7022476196289062, 0.2336745411157608, 1.2037938833236694, 1.8845396041870117, -1.7863103151321411, -0.5745092630386353, -0.9406724572181702, -0.40603742003440857, -0.3392902612686157, -0.2266111671924591, 0.1540757417678833, -0.9464962482452393, -1.2550475597381592, 1.8690383434295654, -1.6024271249771118, -0.9723412394523621, 0.09874071180820465, -1.406754732131958, 0.1363716572523117, -0.05312405154109001, -1.7004759311676025, -0.33792567253112793, -2.2193548679351807, 0.36902928352355957, 1.480969786643982, -0.8712087273597717, 0.349315345287323, -0.3316596746444702, 0.08639131486415863, 0.149277463555336, 0.48015153408050537, -0.8173249959945679, 0.40886959433555603, 0.9206446409225464, -2.0723185539245605, -0.043187130242586136, -0.19412456452846527, 0.21959422528743744, -0.9274632930755615, -0.4672389328479767, -1.3031178712844849, -0.697490394115448, 0.014490880072116852, -0.3823868930339813, 0.3936154842376709, 1.4104992151260376, 0.9094942212104797, -0.9496476054191589, -0.08275259286165237, 1.8353896141052246, -2.104567527770996, 1.215225100517273, 0.12752388417720795, -0.340763658285141, 1.1822627782821655, 1.1723382472991943, -0.5179663300514221, -0.025928920134902, -0.4289409816265106, -1.4070544242858887, 0.17536212503910065, -0.9495998024940491, 1.7010104656219482, 0.40710151195526123, 0.5423937439918518, 0.5635556578636169, -2.4156644344329834, -0.9686171412467957, 0.6695584058761597, 0.6192888021469116, -0.45097482204437256, -0.23124180734157562, -0.4324887990951538, -2.184471607208252, 0.5735125541687012, -1.7237794399261475, -0.9798464775085449, 0.4829779863357544, 0.4992361068725586, -1.5122385025024414, 1.203534722328186, -1.473014235496521, -0.12156618386507034, -0.15839096903800964, -1.9645869731903076, 1.1210098266601562, 0.423742413520813, -1.749550700187683, -0.16257266700267792, -1.0920488834381104, -0.9622361660003662, 1.0496569871902466, -1.2760134935379028, 0.5631281733512878, -1.749169111251831, -0.9199861884117126, -0.7169749736785889, -0.7524663805961609, -0.6574712991714478, 1.9120268821716309, -1.6756629943847656, -0.2591758370399475, -0.24463728070259094, 0.7192588448524475, -1.1891872882843018, -0.17176426947116852, 0.7326728701591492, 1.745053768157959, -0.4235842525959015, -0.6996872425079346, 0.5473126173019409, 0.5563717484474182, -0.999359130859375, 0.6794395446777344, -1.197528600692749, -1.1077094078063965, 0.8415834903717041, 1.374047875404358, 0.4624902009963989, 0.7459486126899719, -0.9016915559768677, -0.8250824809074402, -0.30722758173942566, -0.2876546084880829, -0.10831528902053833, 0.2763221263885498, -0.16054655611515045, -0.6875208020210266, -1.199316382408142, 1.2156200408935547, 0.2349402755498886, 1.3336554765701294, 0.6690564155578613, -0.026446957141160965, -0.31774282455444336, 0.8791599869728088, -1.1434755325317383, 0.2819700539112091, -1.75869882106781, -0.5343624353408813, -1.7834392786026, 1.6403037309646606, 2.4819509983062744, -0.33287766575813293, -0.39720770716667175, 0.04317470267415047, -1.9284547567367554, 1.1375898122787476, -0.7813255786895752, -0.4206666648387909, 0.6455447673797607, 0.5186117887496948, 1.0335625410079956, -0.13586632907390594, 1.4724186658859253, -0.3622185289859772, -1.629912257194519, 0.17477749288082123, -0.22761276364326477, -0.37024348974227905, 0.40141773223876953, 1.4501943588256836, -0.9407867193222046, 0.5971146821975708, -0.272641658782959, -1.8477212190628052, -0.8519208431243896, 0.9958170056343079, 1.624010443687439, -0.0708003118634224, 0.1663445681333542, 1.0173494815826416, -0.9024956226348877, -0.7609105110168457, 0.28716152906417847, 0.0816044956445694, 1.0629500150680542, -1.658600091934204, 0.9424290060997009, -0.5000728964805603, -0.7391362190246582, 0.8240618109703064, 0.1304600089788437, -2.067215919494629, -0.9081681370735168, 0.41503000259399414, 0.29687437415122986, -1.0820637941360474, -0.2994949519634247, -0.22990597784519196, -0.36116823554039, 0.11228957772254944, 1.2220534086227417, -1.0714311599731445, 0.17287972569465637, -1.0869468450546265, 0.609935462474823, -0.6538628935813904, -0.24118004739284515, -1.895089030265808, -2.1335904598236084, 1.148773193359375, 0.31529825925827026, -0.7793566584587097, 1.6238844394683838, 1.51144540309906, -1.4158689975738525, -0.7945459485054016, -0.8899632692337036, 0.7956490516662598, 0.5296173691749573, -0.9430726170539856, 1.3380305767059326, -1.6559280157089233, 1.418929934501648, -2.5096275806427, 0.7692055106163025, 0.4004882872104645, 1.699799656867981, -0.33769962191581726, 0.7145748734474182, -0.6863022446632385, 1.0740649700164795, -0.3780941367149353, -1.5472747087478638, -1.522749423980713, 1.316723108291626, 1.163291335105896, -0.1827518194913864, -1.5716544389724731, 0.17070049047470093, 0.6346469521522522, -0.30373167991638184, -0.28280171751976013, 1.3859810829162598, -1.1876300573349, 1.0131040811538696, 0.1794237196445465, 0.9598661661148071, 0.33207595348358154, -0.815403938293457, -0.49189281463623047, -0.6308754086494446, 0.3326633870601654, -0.7409136891365051, -0.9038876295089722, -1.402439832687378, -1.3459584712982178, 0.21154658496379852, 1.5548919439315796, -0.545250654220581, -0.3860943019390106, 0.026114700362086296, 0.9793773889541626, 1.0901745557785034, 1.4215168952941895, -0.5440188050270081, 0.37075114250183105, 0.609351634979248, -0.5831543207168579, -0.9598556160926819, -0.30063554644584656, -0.33817651867866516, -1.3152964115142822, -0.7731948494911194, 0.649431049823761, 1.0629394054412842, 1.4554840326309204, 0.22036664187908173, -0.2730869650840759, 0.5228873491287231, -0.006077154073864222, 0.7934929728507996, 0.2803812026977539, -2.097761631011963, 1.2808289527893066, 1.4737865924835205, 0.2897469997406006, 2.182371139526367, 0.47926566004753113, -0.5550235509872437, -1.156577229499817, 0.7914552092552185, -1.1924450397491455, 1.651403546333313, 0.5800750255584717, 1.6542272567749023, -0.6137585639953613, -0.602115273475647, 0.7336254715919495, -1.6149101257324219, 0.27489277720451355, 1.0520685911178589, 2.0294101238250732, -0.8266619443893433, -0.5468866229057312, -1.0182229280471802, -0.9593830704689026, 0.22655649483203888, -0.5277708172798157, 0.7799158096313477, -0.7015732526779175, -0.6262437105178833, 0.2517669200897217, 0.383276104927063, -0.019522614777088165, 0.0220484621822834, 0.900464653968811, 0.2148001492023468, -0.36568304896354675, -0.7699590921401978, 0.5361440777778625, -0.7144497632980347, 0.7992016077041626, -1.2578476667404175, 1.0655442476272583, -2.672992467880249, -0.7383872866630554, 1.2723402976989746, 1.038149118423462, 1.313355565071106, -0.47604119777679443, -0.24821245670318604, 1.7568453550338745, -0.5526548027992249, -0.6879692077636719, 2.4217238426208496, 0.06540422141551971, -0.06007549166679382, -1.2973999977111816, -0.6475561261177063, 0.46255791187286377, 2.2999486923217773, 0.784095823764801, 0.7516829967498779, -0.668408215045929, 1.0614322423934937, -0.9018285870552063, 1.1148375272750854, 0.8645476698875427, -0.16304191946983337, -1.1740506887435913, 1.3177944421768188, 1.9561522006988525, -0.47156476974487305, -0.9440742135047913, 2.3436896800994873, 2.4484944343566895, 1.001979947090149, -2.4829986095428467, -0.1340324729681015, 0.5322637557983398, -1.9560638666152954, -0.9258155822753906, -0.8017206788063049, -1.6136341094970703, -0.22581487894058228, -0.30665305256843567, -0.4511388838291168, 0.2760730981826782, -1.7737553119659424, 1.219664216041565, 0.6315661072731018]], "dtype": "float32"}, {"__nd__": [-0.44103315472602844, 0.23493918776512146, -1.6906397342681885, -1.2189322710037231, 0.9923278093338013, -0.6614637970924377, 0.2640202045440674, 1.9830529689788818, -0.5527905225753784, 0.7091231346130371, 0.4142288863658905, 0.034460000693798065, -0.4341181516647339, 1.5289965867996216, 1.1398489475250244, 0.02469046227633953, 1.4687550067901611, 0.4869134724140167, -0.7370758652687073, -0.03284794092178345, 1.6014759540557861, 0.9376375079154968, -0.08642743527889252, -0.09517264366149902, 1.091318964958191, -1.4957036972045898, -1.1292320489883423, 0.34764572978019714, 1.0563801527023315, -1.7422339916229248, -0.26187407970428467, -0.07607155293226242, 0.1264527440071106, 0.2567245364189148, -0.41592517495155334, -1.5362625122070312, 0.5673116445541382, 0.5626386404037476, 0.23619291186332703, 0.2625161409378052, -0.21768014132976532, -0.7112898230552673, 1.2088314294815063, -0.6097069382667542, -0.0041145882569253445, -1.6965306997299194, -0.9726725220680237, -1.3259172439575195, -0.6983516812324524, 0.30804774165153503, -2.201263904571533, 0.918843686580658, 0.4420643746852875, -0.8800573945045471, 0.7731824517250061, 1.008976936340332, -0.6499546766281128, -0.13062073290348053, 0.20888835191726685, 0.9691953659057617, 0.4839820861816406, 0.10670828074216843, -0.34477490186691284, 0.33565568923950195, 0.12687690556049347, 1.5897271633148193, 0.41276437044143677, -0.5108155012130737, 0.987987756729126, 0.46225225925445557, -2.217118263244629, 0.5167000889778137, -0.37082183361053467, -0.7738974094390869, -0.0863904133439064, 0.6028855443000793, -0.4058524966239929, 0.6527589559555054, 1.0606822967529297, 1.0660909414291382, -1.5944610834121704, -0.16530418395996094, 1.8777357339859009, 0.8224616050720215, 0.4953819215297699, 0.3909820318222046, -0.05616195127367973, 0.5263484120368958, 0.019574977457523346, 1.2308605909347534, -0.6647788286209106, -0.029843540862202644, 0.4880690574645996, 0.9610910415649414, 0.6003150939941406, -1.1158208847045898, -1.0324480533599854, -0.33792027831077576, -0.2080123871564865, 0.6248795986175537, -1.3219908475875854, -0.24114960432052612, 1.7058652639389038, 0.49362459778785706, 1.3529376983642578, 0.9561207294464111, -0.664002537727356, -0.12807656824588776, -0.35625746846199036, -0.16884075105190277, -1.1752570867538452, 1.662771463394165, 0.8891196846961975, -1.5714589357376099, 0.20114962756633759, 0.7996275424957275, -0.5257893204689026, 0.11080321669578552, -0.6639103889465332, 1.473190426826477, 0.3213043808937073, 0.27503466606140137, -0.5582753419876099, -0.0027675176970660686, -0.3069595396518707, -0.0764964297413826, -0.9921415448188782, -0.45495539903640747, -0.8316614627838135, 0.7443579435348511, -0.37620654702186584, -0.9524968266487122, -1.0796431303024292, 0.690569281578064, 0.23098914325237274, 0.6496834754943848, -0.9329001903533936, 1.6057029962539673, -0.5799154043197632, -0.2320476919412613, 0.9311168193817139, -1.331933617591858, 0.5153399705886841, 0.7651918530464172, 0.02774813584983349, -0.5072162747383118, 2.3201277256011963, 1.8632911443710327, 0.39104771614074707, -2.614325523376465, 0.4292173385620117, -0.12365812063217163, 0.44774147868156433, -0.22839167714118958, 0.2908841669559479, -1.3178085088729858, 0.59461909532547, -0.1451171487569809, -0.05030716955661774, -1.2802939414978027, 1.9766077995300293, 0.7051751613616943, -2.966092586517334, 0.3143123686313629, -1.0012818574905396, -0.6478249430656433, -0.5622599720954895, 0.6290562152862549, 1.1960963010787964, 1.7215659618377686, -0.6510887742042542, 0.17043446004390717, -0.21937322616577148, 0.4052249491214752, 0.14815643429756165, 0.13272368907928467, 0.8383271098136902, -1.127453088760376, 0.39507371187210083, 0.732038676738739, 0.17270910739898682, -0.8817381262779236, -2.060394763946533, -1.1719111204147339, -0.9304456114768982, 0.2208794355392456, -0.49871063232421875, -0.1498289406299591, 0.7359205484390259, 0.8130501508712769, 1.6208449602127075, -0.19431006908416748, -1.391289472579956, -1.0702509880065918, 1.1420271396636963, 1.028257966041565, 0.8861258029937744, -0.7108754515647888, 1.5061160326004028, 1.7525871992111206, 1.1985645294189453, 0.4936354160308838, 0.6170620322227478, -1.100037693977356, 0.49349913001060486, -1.0128347873687744, 0.4121890962123871, 0.32463523745536804, -0.19728073477745056, -0.6229406595230103, -0.02457604371011257, 0.6732970476150513, 0.1401481032371521, 0.7812932729721069, 1.421134114265442, 0.567548930644989, 1.5480133295059204, -0.065631203353405, 0.13920356333255768, 2.1040596961975098, 1.9605207443237305, -0.44866880774497986, 0.3612931966781616, -1.5496619939804077, 0.7813711762428284, -1.0871189832687378, 0.4332919120788574, 1.420931100845337, -0.40311965346336365, -1.4576112031936646, 0.019829321652650833, -0.19499915838241577, -1.9801898002624512, 1.8386929035186768, 1.0285630226135254, -0.09702567756175995, -0.5379186272621155, 1.027491569519043, -0.5531673431396484, -1.1307567358016968, 1.5586885213851929, 1.0455150604248047, 0.5277222394943237, -2.1052451133728027, 1.2638182640075684, -0.22398395836353302, -1.0883777141571045, -1.530863642692566, -0.45961952209472656, -0.7557187080383301, 0.7996960282325745, -0.938308596611023, 0.46514374017715454, 0.14085237681865692, -0.3798443078994751, 0.5307785272598267, -0.713828444480896, -1.2564674615859985, -0.7207671403884888, -0.396883487701416, -0.06570091098546982, 0.6951793432235718, -0.02704099752008915, 0.5435996651649475, 0.03565279021859169, -0.1597157120704651, -1.1785715818405151, -0.055957723408937454, -0.3989284932613373, -1.1769143342971802, -0.7480330467224121, -0.810377299785614, 1.9358360767364502, 0.8943979740142822, 0.6624478697776794, -1.0942034721374512, -0.8223817944526672, -1.0927395820617676, 0.2278815507888794, 0.4938332438468933, 0.7533844113349915, -0.013696075417101383, 0.013526640832424164, -0.6881240606307983, 0.0788402333855629, 0.32620617747306824, 1.1418861150741577, 0.5563715100288391, 1.226502776145935, 0.6523852348327637, -1.3546679019927979, 0.4020299017429352, -0.38036689162254333, 0.15675054490566254, -0.8921521306037903, 0.6527466177940369, 1.7358249425888062, 1.5445234775543213, -0.5231235027313232, -0.04223125800490379, -0.8143823146820068, -1.0933949947357178, 0.5909915566444397, 1.7647058963775635, -2.535393714904785, 0.28646618127822876, 1.3923381567001343, -1.8762435913085938, -0.42809438705444336, 0.14331628382205963, 1.8748419284820557, 0.5284306406974792, 1.4445762634277344, 0.8255718350410461, 0.6595832705497742, 0.11590531468391418, -1.0277180671691895, -1.724924087524414, 0.41149410605430603, 0.6572527289390564, -1.9832634925842285, -1.3597768545150757, 0.37414902448654175, 0.1717045158147812, 0.07696352899074554, -0.10955695807933807, 0.5044222474098206, -0.7149295806884766, 0.27587297558784485, 0.001086734002456069, -0.9473823308944702, -0.18174102902412415, -0.0008477323572151363, 0.5188590288162231, 0.8494660258293152, 0.2399216741323471, -0.07677840441465378, 0.2940359115600586, 1.7786328792572021, 0.2925930321216583, 1.970157265663147, 0.8686708807945251, 0.20121492445468903, 0.2226972132921219, 1.5160014629364014, 0.31104347109794617, -0.49710792303085327, 0.4260319173336029, -0.10362625867128372, 0.7480175495147705, 0.4191402494907379, 0.4238538146018982, 2.0570411682128906, -0.08686734735965729, -0.024104595184326172, 0.38105228543281555, -0.5114617943763733, -0.11225803941488266, 1.5122017860412598, 0.6834185123443604, -0.13555438816547394, 0.6359164714813232, 0.6703822612762451, -0.9029207229614258, 0.7686150670051575, -0.5148743987083435, -0.8007628321647644, -0.0765896737575531, -0.11591129004955292, -0.6573725342750549, 2.5271706581115723, 0.48182567954063416, 0.5041085481643677, -0.978512167930603, -2.337357759475708, 0.5792410373687744, 2.533616065979004, -0.7871314883232117, 0.2204395830631256, -1.05028235912323, 0.15000812709331512, 0.6233236193656921, -0.13099795579910278, -0.38158199191093445, -0.3551202416419983, -0.26440516114234924, -0.39437711238861084, 0.37748008966445923, -0.7332480549812317, -0.30622467398643494, -2.5203135013580322, -0.6196824908256531, -0.327599436044693, -0.32881420850753784, -0.5187060832977295, -0.7678900361061096, -0.9810985326766968, 0.2330930233001709, 0.6784724593162537, -1.5808879137039185, -0.18063360452651978, -0.5844405293464661, 2.765953302383423, -0.06893151253461838, -0.9796797633171082, 0.47026777267456055, 1.1539310216903687, 1.2527576684951782, -0.12659554183483124, -0.8776838779449463, 0.07629765570163727, 1.5780277252197266, -0.7848972678184509, 1.576424479484558, 1.0006426572799683, -0.82440584897995, 1.3544753789901733, 0.9702009558677673, -0.12909424304962158, 0.5578962564468384, -1.7621222734451294, 1.8552765846252441, 1.614052414894104, -0.5428274273872375, -0.26925042271614075, -0.8119446635246277, 1.1641680002212524, 0.0015790744218975306, 0.9456561207771301, -2.337272882461548, -0.1239735409617424, -0.4138963520526886, 0.4645332992076874, -0.32271066308021545, 1.1462643146514893, 1.9166580438613892, 0.3923858106136322, -1.5794123411178589, -1.0613536834716797, -1.1936509609222412, 1.244725227355957, -0.5037838816642761, 1.501691460609436, 0.23809318244457245, 1.3708919286727905, 0.5810867547988892, -1.1586294174194336, 1.744949221611023, -0.5249796509742737, -1.4870773553848267, -2.576080083847046, -0.7233559489250183, 1.0732516050338745, -1.02244234085083, -0.25130707025527954, -1.0499687194824219, 0.583039402961731, 0.925575852394104, 0.46253952383995056, 1.2648465633392334, -1.0357887744903564, -0.29914048314094543, 0.9747045636177063, 0.6070829033851624, 1.8377279043197632, -0.23948539793491364, 0.31766533851623535, -1.3076412677764893, -0.1432744860649109, 0.10434100776910782, -0.885368824005127, -0.48949533700942993, 0.319232314825058, -1.1435043811798096, -1.724437952041626, 0.20928777754306793, -0.7320762276649475, -1.1171536445617676, 0.36663368344306946, 1.6582636833190918, 0.06262660026550293, 0.1492718756198883, 0.12466811388731003, 1.6194957494735718, -0.009953048080205917, 0.05137025937438011, 1.9172781705856323, -1.5546276569366455, 2.0036442279815674, 0.7231225967407227, 0.22336794435977936, 1.9980814456939697, -0.8149226903915405, -0.6460026502609253, -0.27725932002067566, -1.524260401725769, -0.16041405498981476, 0.03542008250951767, -0.6840585470199585, 1.3182860612869263, -1.2370023727416992, -0.9162969589233398, -0.1658077836036682, 0.5652161240577698, 1.6943087577819824, 0.12303011119365692, 0.7971246838569641, -0.05660521239042282, -0.023941440507769585, -0.1402777135372162, 1.2841525077819824, -0.43727752566337585, 0.09512156248092651, -0.26076433062553406, 0.31488990783691406, 0.5400144457817078, 0.7473050951957703, -0.3160783350467682, 0.9793991446495056, 0.2502601146697998, 0.5803783535957336, 0.15280689299106598, 0.5442591309547424, 1.0313090085983276, -0.018456468358635902, -0.9937664270401001, 0.5173094868659973, 0.17016983032226562, 0.8265858292579651, 1.805578351020813, 2.0436818599700928, -0.6308936476707458, -0.27577969431877136, -0.8180757164955139, 0.05900709331035614, 1.5298941135406494, -1.0181444883346558, 1.9668807983398438, 1.2881230115890503, -0.1265258640050888, -0.04813133180141449, 1.01630699634552, -0.2695619463920593, -0.3282621204853058, -2.759357452392578, 1.538686990737915, 0.3957861065864563, -0.7073853015899658, -0.10470321029424667, 0.6100811958312988, -0.5211904048919678, 2.2052161693573, 0.015177898108959198, -1.215158224105835, 0.16870494186878204, -0.6919557452201843, -0.7823135256767273, 0.5549191236495972, -0.21236130595207214, -0.05995536223053932, 0.008662414737045765, 1.3230468034744263, -0.41277071833610535, -0.6241908073425293, 0.3134855329990387, -1.007187008857727, 0.43992748856544495, 0.07632023096084595, -0.8335655927658081, -0.5336195826530457, 1.135750651359558, -0.5430839657783508, -0.20257890224456787, -0.057875342667102814, 1.1544486284255981, 0.36433303356170654, 0.27211102843284607, 1.9365882873535156, -0.3000905513763428, -0.42806339263916016, 0.4089372456073761, 1.0396370887756348, -0.8587867021560669, -1.0593172311782837, -2.8093199729919434, 1.0557843446731567, -2.344391345977783, 0.40683403611183167, 0.951123833656311, 2.614790678024292, -0.8283390998840332, -0.5274519324302673, -0.36104801297187805, -0.3417525291442871, -2.137584686279297, -0.7330798506736755, -0.41780346632003784, 0.9606569409370422, 1.40798020362854, 0.10054344683885574, 0.8526062965393066, -0.6678473949432373, -1.1575100421905518, 0.5369130969047546, -0.29269564151763916, -1.6692564487457275, -0.4829699397087097, 0.23072801530361176, -1.2113062143325806, -1.9578919410705566, 1.0194121599197388, 0.030626263469457626, -0.2467907816171646, 0.30576571822166443, 1.2297087907791138, 0.20613466203212738, 1.3895291090011597, -0.30118414759635925, -1.1442980766296387, -0.9898837208747864, -0.03539518266916275, 0.8118205070495605, 0.41746827960014343, -0.94803786277771, -0.30474135279655457, -0.46318134665489197, 1.0024775266647339, 0.09048032760620117, -2.2649126052856445, -0.9688718318939209, -1.1656126976013184, 0.32316890358924866, -0.09322638809680939, -0.9058597087860107, 1.4019721746444702, -0.04619240760803223, 0.5013071298599243, 0.42168694734573364, -0.4143376350402832, -1.5120049715042114, -0.3720746636390686, -0.6734548807144165, 0.06549713760614395, 0.599477231502533, -1.4862273931503296, 0.1935907006263733, 0.8037444949150085, 0.34230637550354004, -0.9972925782203674, 1.1760475635528564, 1.2297145128250122, 1.1839252710342407, -0.1759842187166214, -0.6795303821563721, -0.25902146100997925, 1.4335302114486694, 1.423664927482605, -0.7402295470237732, 1.3509174585342407, 1.1721618175506592, 1.2365012168884277, 1.0135291814804077, 0.16489067673683167, 0.9630834460258484, 1.4881682395935059, -0.012093714438378811, -0.5478277802467346, -0.2827736437320709, 0.8716135621070862, -0.49500903487205505, -0.3706973195075989, 0.8481641411781311, 0.4481869339942932, -1.86957848072052, -0.560987114906311, -1.0307239294052124, -0.6849774718284607, 0.9488183259963989, 0.8446772694587708, -0.8989212512969971, -1.2614340782165527, -0.006003763526678085, 0.5814683437347412, -0.2285398244857788, -2.1529505252838135, -1.3103734254837036, -0.6093531250953674, 0.17552006244659424, -0.376478374004364, -1.2478446960449219, -0.17665110528469086, 1.464400053024292, 0.21793751418590546, 1.317152738571167, 0.5247659683227539, -0.7668901681900024, 0.1359187811613083, -0.8526813387870789, -0.6518886685371399, 0.14354188740253448, -1.2837456464767456, 0.6160770058631897, -0.8055242896080017, -0.5813505053520203, -0.010614918544888496, 1.0363825559616089, -0.060447223484516144, 1.5359601974487305, -1.30393385887146, 0.6245251297950745, 0.3883151412010193, -0.14269722998142242, 0.6402106285095215, -0.12045825272798538, 0.47253286838531494, 1.0573993921279907, -0.015331726521253586, -1.7450003623962402, 0.47632527351379395, -0.6190846562385559, 0.8025222420692444, 0.7248510122299194, -0.8170453310012817, -2.323734998703003, -0.088223896920681, 1.1396459341049194, 0.39493653178215027, -0.7508224248886108, -0.818135678768158, -0.1356716901063919, 1.8897316455841064, -0.29248538613319397, 0.19629411399364471, 0.6741957664489746, 0.6977965831756592, 0.814333975315094, 1.577652096748352, 0.252078115940094, -0.5167623162269592, -0.8175715804100037, 1.4177119731903076, -1.2836426496505737, 0.6916595697402954, 0.07020211219787598, 1.2106966972351074, -0.14384111762046814, -0.21598798036575317, 1.6135506629943848, 1.6039366722106934, -2.603490114212036, -0.2815997898578644, 1.5260642766952515, -0.06383009254932404, 0.5418896675109863, -0.9427628517150879, -0.8837070465087891, -0.48321881890296936, -0.8588437438011169, -1.9970736503601074, 0.22583849728107452, 0.9044348001480103, -0.9935599565505981, 0.8222929239273071, 0.46612071990966797, -0.6925736665725708, -1.6387685537338257, 2.8365046977996826, -0.8310412764549255, -0.6543004512786865, 0.7088952660560608, 1.0487290620803833, 0.6078177690505981, -1.0291789770126343, -1.446952223777771, -1.3357698917388916, -0.07266759127378464, -0.5782569050788879, 0.549237072467804, 0.1369933784008026, -0.6435763239860535, -1.3422446250915527, 0.8099871277809143, 1.25771164894104, -0.8893311023712158, 0.9889426231384277, 2.7792909145355225, 0.49766939878463745, 0.42575210332870483, 2.441265821456909, 1.0307683944702148, 0.02208702452480793, 0.5402640700340271, -1.7743090391159058, -1.020238995552063, -0.5427252054214478, 1.4670518636703491, -0.5195249319076538, 0.7450499534606934, -0.023962946608662605, -0.8659527897834778, 0.460197389125824, 1.2357133626937866, -1.4244447946548462, -0.26774492859840393, -0.22660194337368011, -1.0610244274139404, -2.6879031658172607, 0.49491360783576965, -0.5443214178085327, -0.14987745881080627, 0.3696577847003937, -0.7841527462005615, 0.32645779848098755, 0.7523517608642578, -0.24490448832511902, 0.7277552485466003, -1.27370023727417, 0.6975958347320557, -0.6810875535011292, 0.512356162071228, -0.6253291964530945, 0.6780496835708618, 0.7739732265472412, -0.7806910872459412, -0.5134603977203369, -0.0008822509553283453, -0.45502662658691406, 0.996649980545044, 0.17944519221782684, 0.17703880369663239, -0.9227136373519897, -0.5616950392723083, -1.1154615879058838, -0.09135252982378006, 0.6722027063369751, -2.8971662521362305, 0.33352866768836975, 0.4203087389469147, 0.4321543574333191, -0.5449143052101135, -0.9951632022857666, -0.02921028435230255, -0.3432044982910156, 0.0026409493293613195, 0.9502643346786499, -1.3453705310821533, -1.0562288761138916, -1.004934310913086, -0.6612845063209534, 0.060649968683719635, 1.1244404315948486, -1.6275980472564697, -0.23680591583251953, 0.18698474764823914, 0.6556572318077087, -1.607298493385315, -1.0280699729919434, -0.592465877532959, -0.6930694580078125, -0.5348808169364929, 0.3732965588569641, -0.025750162079930305, -1.7221215963363647, -0.8565133213996887, -0.5392568707466125, -1.0782437324523926, -0.16246119141578674, -0.7099012732505798, -1.864601492881775, -1.472900629043579, -2.2560336589813232, -0.32656243443489075, -1.8735159635543823, 1.2191768884658813, 0.37855061888694763, 0.4690651297569275, 1.5891597270965576, -0.16007381677627563, 0.024769918993115425, 2.098846197128296, -0.15780630707740784, 0.23445796966552734, -0.11374318599700928, -0.555134654045105, 2.220123767852783, 1.3916200399398804, -0.03917831927537918, 0.6607758402824402, -1.418397307395935, -0.8612778186798096, 0.49026745557785034, 0.6092459559440613, -1.3685686588287354, -0.05490845441818237, 0.9502396583557129, -0.9669862985610962, -1.0845911502838135, -0.910366952419281, 1.1143183708190918, -0.6848163604736328, -0.4269877076148987, 0.04278743639588356, 0.3081929683685303, -0.45591306686401367, 0.14915256202220917, -1.1459842920303345, 0.016219619661569595, -0.03705527260899544, 0.8701305389404297, -0.5622316598892212, -0.6887776255607605, -0.16576120257377625, -0.12964345514774323, 1.8828930854797363, -0.5629864931106567, -0.8107797503471375, -0.443451851606369, 0.9911524057388306, 1.2987946271896362, 0.18797717988491058, -0.4942210018634796, 0.007301672361791134, -1.02798593044281, 0.5339751839637756, -0.7441297769546509, -2.2936172485351562, -0.38869789242744446, -0.4702875018119812, 0.8716427087783813, -0.983154296875, 0.7157002091407776, -0.4883385896682739, 0.07844997197389603, -0.41325122117996216, -1.245398998260498, -0.37922152876853943, -0.726972222328186, 1.8414530754089355, -1.1932401657104492, 0.7094787955284119, -1.1798532009124756, -1.1311213970184326, -0.1950378268957138, -0.5349610447883606, 0.47399255633354187, -0.3207748234272003, -0.0801202729344368, -1.3521125316619873, 1.9892715215682983, 0.8291581273078918, 0.18733753263950348, 0.5141910910606384, -1.3042659759521484, -1.45210599899292, -0.6595157384872437, 2.27536678314209, -0.11179781705141068, -0.6226862668991089, 0.13932563364505768, -0.19328156113624573, 0.3301130533218384, -1.5870672464370728, 0.5588037371635437, 1.2210115194320679, 0.9580797553062439, 0.37817811965942383, 1.4244499206542969, 0.5218526721000671, -1.3878130912780762, -0.21950246393680573, 0.9987958073616028, -0.413083553314209, -0.5067564249038696, 0.7627767324447632, -0.11358095705509186, 1.0331401824951172, 0.913097620010376, -0.20568767189979553, 0.8483346700668335, 0.9352493286132812, 0.7145811915397644, -0.08985541760921478, 0.9410429000854492, 2.098942995071411, -0.799472987651825, -0.6456642150878906, 0.691668689250946, -0.1847531646490097, -0.586747944355011, -0.14393684267997742, -0.7886923551559448, -0.4496065676212311, -0.26392966508865356, 1.5110254287719727, 0.2854042947292328, 0.7039976119995117, 0.3472563326358795, 0.10713718086481094, -0.650925874710083, -1.8781368732452393, 1.9840245246887207, 2.057525634765625, 0.601341724395752, 0.3573574423789978, 0.5175559520721436, -0.37990885972976685, -1.2751166820526123, 1.779090166091919, 2.498943328857422, 1.2094162702560425, 2.1619558334350586, 0.7545442581176758, -0.9656997323036194, 0.4780150055885315, -0.8521298170089722, -0.6208733916282654, 0.6581775546073914, -0.8678286671638489, 2.070333480834961, -1.4476258754730225, -0.19152912497520447, 1.4789842367172241, -0.1676168292760849, -0.6725142598152161], "dtype": "float32"}, {"__nd__": [-1.50136399269104, -0.6112882494926453, -0.6650313138961792, 0.9068467617034912, -0.5140621662139893, 0.004388878587633371, 0.7724672555923462, -0.4945254325866699, 0.6312441825866699, -0.4906735420227051, -0.5453559160232544, 0.7961841821670532, 0.1298108547925949, -0.6222929358482361, -0.8658953905105591, -0.5464357137680054, -0.4868384897708893, -1.586365818977356, -1.307375192642212, 1.2008445262908936, 2.9395601749420166, 0.2219160795211792, -0.13172291219234467, -2.5531973838806152, -1.0215662717819214, 0.465450257062912, 0.7760241031646729, -0.014429054223001003, -0.10088317096233368, -0.5424010157585144, -0.7656617760658264, -1.0993022918701172, -0.790265679359436, -1.5810359716415405, 0.9716468453407288, -1.5113781690597534, -0.02522450126707554, 1.2782108783721924, 0.609053373336792, 0.09482311457395554, -0.024699440225958824, 0.568433403968811, 0.281259685754776, 0.35373640060424805, 0.4422808587551117, -0.2873193919658661, -0.8448632955551147, -1.1943135261535645, -0.2517659366130829, -1.939791202545166, 0.2658514380455017, -0.2739908993244171, -0.5816128253936768, 0.3217217028141022, 1.2729181051254272, 0.19437779486179352, 1.428317666053772, 1.8926031589508057, 2.3980538845062256, 1.3065769672393799, 1.0378637313842773, -0.5475444197654724, 0.7905088663101196, 3.3466386795043945, -0.7446918487548828, 2.0614185333251953, 0.38978204131126404, 0.7737191319465637, 0.3080716133117676, -0.1519865244626999, -0.7791056632995605, -0.3151974081993103, -0.880608320236206, 1.4177154302597046, -0.0664159506559372, -0.9798212051391602, 0.7406102418899536, -0.942714512348175, -0.6339716911315918, -0.3738866448402405, -0.19088394939899445, 0.025537261739373207, 1.1793041229248047, -0.24232903122901917, 1.164211630821228, -0.9795229434967041, 1.2347853183746338, 0.14213117957115173, 0.5094099044799805, -0.08533596247434616, -0.27688831090927124, -1.9511100053787231, -0.6740393042564392, -0.46863994002342224, 0.12051328271627426, -0.003120419569313526, -0.35524195432662964, -0.5793856382369995, 0.8762177228927612, 1.076075553894043, -1.3593552112579346, 0.1774669885635376, 0.3940429389476776, 1.0060149431228638, 0.9581639766693115, 0.5465816259384155, 1.4516702890396118, 0.602303683757782, 0.6854889988899231, -1.492685317993164, 0.5034151673316956, -0.562587320804596, 0.6508200764656067, 0.9178826808929443, -1.4083271026611328, 0.4606681168079376, -0.7465485334396362, 0.6592327952384949, 2.0188724994659424, -0.8095638751983643, -0.10005185753107071, -0.3122703731060028, -0.5040574669837952, -0.7266786694526672, -0.8837792873382568, 0.360400915145874, -0.4280446767807007, -0.09989467263221741, -0.6602969169616699, -0.8488150835037231, -0.27405714988708496, 1.3255252838134766, 0.9498072266578674, -0.5726892352104187, 1.5736063718795776, 1.508753776550293, 1.0680404901504517, -0.26564139127731323, 0.07021232694387436, 1.8626638650894165, -0.7939965724945068, 0.08712607622146606, -1.7523746490478516, 0.506544828414917, 1.1041436195373535, 1.757004976272583, -1.9956862926483154, 1.130020260810852, -1.3218071460723877, -0.12843184173107147, -0.412311851978302, 0.7184372544288635, 1.3089083433151245, 1.8612290620803833, 0.25708118081092834, 0.6370660662651062, -0.8397494554519653, -0.587897777557373, -0.39592552185058594, 0.4845767319202423, 0.733925998210907, 0.6146472692489624, -0.7886984944343567, 0.9402320981025696, 0.3221813142299652, -0.49796026945114136, -1.2326277494430542, 0.9906742572784424, 1.5052053928375244, 1.3192203044891357, -1.0093481540679932, -0.9104422926902771, 0.8468510508537292, -0.3920503556728363, 0.1914965957403183, 1.3350692987442017, 0.747215986251831, 0.6797816157341003, 0.6689844131469727, -1.2588725090026855, 1.0414643287658691, 1.5886305570602417, -1.1263542175292969, 1.39150071144104, -0.8588804006576538, -0.6917028427124023, -0.817956268787384, -2.527971029281616, -1.1565319299697876, 0.16838014125823975, 1.2826908826828003, 0.4751821458339691, 0.9389587044715881, -0.5069663524627686, 0.014263705350458622, 0.30748191475868225, 1.4077242612838745, 0.8641032576560974, -1.2825521230697632, -0.41916555166244507, -1.0242174863815308, -0.8247912526130676, 1.7486571073532104, -0.6710984110832214, 0.6501929759979248, 0.6515043377876282, 1.2072558403015137, 1.429484248161316, 0.6638089418411255, -1.6318132877349854, 1.0369714498519897, -0.05467313900589943, -1.1262420415878296, -2.182732105255127, 0.5622863173484802, 0.8352271914482117, 0.639199435710907, 0.4877806603908539, 0.09181301295757294, -0.10502516478300095, 1.3075627088546753, 0.7611156702041626, -0.49709656834602356, 0.1274549812078476, -0.39384084939956665, 0.9368686079978943, 0.22147342562675476, -0.7732417583465576, 0.2714669704437256, 0.05126170068979263, -0.8537804484367371, 0.686632513999939, 1.0504112243652344, -0.3315582871437073, 2.596099853515625, -0.5763941407203674, -0.6465361714363098, 1.0632582902908325, 0.20419669151306152, -0.307557612657547, 0.7385601997375488, -0.6494280099868774, 0.7721420526504517, -0.24406187236309052, 0.11633187532424927, -1.2626597881317139, -0.29624080657958984, -0.17966799437999725, 0.2047303467988968, -1.1042242050170898, -0.06438476592302322, -0.30799078941345215, -1.0567222833633423, -0.01386642549186945, 0.2849714756011963, -0.4752347767353058], "dtype": "float32"}, 1.0, 0.0], "e": {"__nd__": [12.619933128356934, 23.728355407714844, -40.0255126953125, 4.2694525718688965, -50.18730163574219, 67.95511627197266, -2.4129114151000977, -29.118770599365234, -35.07268142700195, 51.73584747314453, 13.126694679260254, -14.604856491088867, -31.940223693847656, -26.24374771118164, 17.65118980407715, -4.095952033996582, 44.440948486328125, -43.798824310302734, -17.494247436523438, 5.18372106552124, 30.12490463256836, -16.34258270263672, 0.705280601978302, -15.358407974243164, -5.338656902313232, 33.261077880859375, 5.822678565979004, 11.451595306396484, -3.944596767425537, 62.23575973510742, 9.358431816101074, 86.74109649658203, -7.447417736053467, -20.63701629638672, -1.6251814365386963, 25.454044342041016, -5.718778133392334, 10.747989654541016, 0.8646664619445801, 82.13581848144531, 10.340949058532715, 33.47078323364258, 28.090641021728516, -14.321063995361328, 88.82027435302734, -33.40066146850586, -37.09351348876953, 64.3719711303711, -4.979682445526123, -18.49224281311035, 21.16495704650879, -4.156874179840088, -16.953983306884766, 30.593549728393555, 21.889747619628906, 14.038022994995117, -2.065121650695801, 24.724864959716797, -16.09317970275879, -32.91133499145508, 51.30876159667969, 17.706268310546875, -68.24674224853516, -6.688987731933594, -80.62202453613281, 36.35198974609375, 50.27926254272461, 5.610586166381836, -0.7802076935768127, -65.79931640625, -12.887327194213867, -13.217885971069336, -40.02275466918945, 4.986350059509277, 6.488672733306885, 8.485894203186035, -17.50213623046875, -9.257441520690918, -16.914716720581055, 20.6153621673584, 17.862516403198242, 45.91627502441406, 2.927823066711426, -38.90971755981445, 16.156221389770508, 11.456266403198242, -18.352636337280273, 4.627854347229004, 23.162803649902344, -37.52412796020508, -23.234355926513672, -0.08672155439853668, 21.00119400024414, -54.509544372558594, 22.183673858642578, 49.692195892333984, -61.79802322387695, -27.47171401977539, -35.53455352783203, 21.505844116210938, -52.14252471923828, 19.60637855529785, 20.800613403320312, 61.61417007446289, -44.8300666809082, -3.7903976440429688, -46.66356658935547, 26.985776901245117, 56.31061935424805, -13.592201232910156, -28.462867736816406, -13.162211418151855, -39.13286209106445, 31.496883392333984, 10.61164665222168, 23.034149169921875, -27.4915828704834, 48.77961349487305, 31.52159881591797, 42.50420379638672, 44.795867919921875, -1.0993844270706177, -19.411245346069336, 10.089768409729004, 57.67506408691406, 15.670406341552734, 9.508243560791016, 19.685914993286133, -11.34722900390625, 40.147186279296875, -31.45513916015625, -26.737144470214844, 20.85899543762207, 39.92664337158203, -44.44098663330078, -44.9967041015625, 50.181400299072266, 21.113554000854492, -2.4605772495269775, -46.50489807128906, -17.03083610534668, 38.66475296020508, 11.977095603942871, 12.677781105041504, 32.499183654785156, -55.30989456176758, -14.027284622192383, -28.217058181762695, -13.517668724060059, -40.49062728881836, 33.423912048339844, 13.887469291687012, 41.0297966003418, 27.521198272705078, -1.2477225065231323, 6.070345401763916, -9.91274356842041, 9.729179382324219, 47.747005462646484, -7.308529853820801, -22.154632568359375, 42.541255950927734, 50.784908294677734, 26.31924819946289, 5.726059436798096, 33.30723571777344, 8.352783203125, -66.32746887207031, 26.229551315307617, 88.99072265625, -1.4815086126327515, -49.97308349609375, -0.3315217196941376, 25.069908142089844, -62.165191650390625, 27.045272827148438, -10.575512886047363, -8.41792106628418, 7.4381184577941895, 67.13261413574219, -4.1575727462768555, -36.592926025390625, -16.869003295898438, -37.53667449951172, 9.054895401000977, -54.529869079589844, -1.335264801979065, 49.116111755371094, -13.60546875, 30.707304000854492, 22.6971378326416, 19.1014404296875, -27.282609939575195, -58.4112548828125, -13.222580909729004, 35.339134216308594, -27.975717544555664, 32.15616226196289, -55.4961051940918, 26.30078887939453, 26.16050148010254, -23.1192684173584, 0.8922227025032043, -23.19627571105957, 12.398262023925781, -31.94845199584961, -44.45330047607422, 50.830726623535156, -25.283180236816406, 28.328311920166016, -54.09055709838867, 49.61711120605469, 9.142192840576172, -25.8450870513916, -23.554473876953125, 16.759944915771484, 12.773921966552734, -13.36796760559082, -7.574563026428223, -3.742255449295044, 0.9987689852714539, 5.298710346221924, -2.7087082862854004, 50.192501068115234, -12.416576385498047, 10.721805572509766, 25.141813278198242, -20.514978408813477, -26.676685333251953, 8.2358980178833, 34.83108139038086, 56.54090118408203, -28.33347511291504, -2.7423105239868164, -25.04866600036621, -34.95224380493164, 17.21438980102539, -26.206340789794922, 16.03676986694336, 58.11789321899414, -26.211694717407227, -39.16517639160156, -1.407669186592102, 71.12530517578125, 4.4580583572387695, -39.51744079589844, 12.839428901672363, -7.6314921379089355, 17.810941696166992, 21.970935821533203, -21.87856674194336, -14.70098876953125, -55.6709098815918, -23.402711868286133, -49.60519027709961, -20.98848533630371], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.28290170431137085, -0.09348408132791519, 0.010599581524729729, -0.1043354943394661, 0.08369965851306915, 0.947405219078064, -0.16577455401420593, -1.38880455493927, -0.9224609732627869, -1.9084457159042358, -0.11027782410383224, 0.2264697253704071, -0.7884863615036011, -1.8432470560073853, -0.2195603847503662, -1.0228266716003418, -0.8038511872291565, 0.2265259176492691, -0.2273140251636505, 0.08362555503845215, -1.4093859195709229, 0.9541337490081787, -0.48550844192504883, 0.7899212837219238, -1.399621844291687, -0.2406207025051117, -1.7358901500701904, -1.2334390878677368, 0.4611804187297821, -0.6304500102996826, -2.2042324542999268, 0.0544557049870491, 1.0205352306365967, 0.6306273937225342, 0.9263608455657959, 0.0042758509516716, -0.6822315454483032, 1.9147989749908447, -0.6053464412689209, 0.27142906188964844, 0.1707809567451477, -0.22733819484710693, -0.24085254967212677, 0.3966371417045593, -2.248910903930664, 0.8934399485588074, 1.1684584617614746, 0.6031213402748108, -0.0024712211452424526, -1.54184091091156, -0.2466403841972351, -0.13319829106330872, 0.21823151409626007, 0.8743026852607727, 0.6170030832290649, -2.426217794418335, -1.994777798652649, 0.3330657184123993, -0.044005632400512695, 1.516852617263794, 0.28928887844085693, 0.37209486961364746, 0.46741533279418945, 0.6745545268058777, -0.6847720742225647, 1.2821422815322876, 0.6713248491287231, -0.23029093444347382, -0.09836624562740326, 2.0736827850341797, -1.3246634006500244, -0.9902698397636414, -0.5261669158935547, 0.7658414244651794, -1.6723710298538208, 0.03283657506108284, -0.8964371085166931, -1.683423399925232, 2.2247209548950195, 1.2901229858398438, 0.79227215051651, -1.0945522785186768, 0.7523893713951111, -0.17206457257270813, 2.049806594848633, 0.37276414036750793, -0.5720477104187012, 0.7944099307060242, 0.8765156269073486, 1.0434958934783936, -0.5784544944763184, 1.2633552551269531, 1.3347283601760864, 0.06927449256181717, 2.0907135009765625, 0.4157498776912689, 0.1057782992720604, -0.3954330086708069, 0.5315931439399719, -0.2956382632255554, 1.4128644466400146, 0.9116499423980713, 1.0072834491729736, -0.5662503242492676, -0.4792952835559845, -1.2907346487045288, -1.0045969486236572, -0.376120388507843, 0.5829335451126099, -1.4765909910202026, -0.21420294046401978, -0.7727015018463135, 0.14149802923202515, 1.3750604391098022, -0.005238300655037165, 1.2128263711929321, -0.1681603491306305, -0.14170032739639282, 0.41154566407203674, 0.3447803258895874, 0.44824492931365967, 0.021805189549922943, 1.2106493711471558, -0.2360561341047287, -0.9217420220375061, -0.7516693472862244, -0.009323269128799438, 0.9781551957130432, 1.6128462553024292], [1.8209017515182495, -0.1511382907629013, -0.3743586838245392, -0.10604511946439743, -0.5277441143989563, -0.12327113002538681, -1.0416795015335083, 0.7750478982925415, -0.005005387589335442, -0.6064877510070801, -1.1497523784637451, 0.5384448766708374, 0.3542843163013458, -0.6832561492919922, -0.8466500639915466, 1.5193604230880737, 0.26888543367385864, -0.507187008857727, -1.1049420833587646, 1.8042073249816895, -1.3754322528839111, 0.4252869188785553, -0.04683811962604523, 1.0470658540725708, -0.02677508071064949, 0.9157885909080505, 0.3767102360725403, 2.4965786933898926, -0.5200501084327698, -0.7708786129951477, -2.2548632621765137, -0.6794602274894714, -0.8738503456115723, -0.17381227016448975, -0.489026814699173, 0.8191713094711304, 1.398005485534668, -0.39736804366111755, -0.6520232558250427, -0.020787786692380905, -1.2997148036956787, 0.7526028156280518, -1.4029781818389893, -2.206293821334839, -0.5995545387268066, -0.18054546415805817, 1.2356632947921753, -0.20557762682437897, 0.8800191283226013, 0.8608137369155884, 0.4201194941997528, 0.23831985890865326, 0.0984133630990982, 0.14354076981544495, 0.8391587138175964, 1.2093538045883179, 1.0733247995376587, 1.4993244409561157, 0.7190451622009277, 1.2314481735229492, 0.12774448096752167, -0.617463231086731, 0.26859739422798157, -1.1814826726913452, -1.095137119293213, 0.12066122889518738, -0.5384556651115417, -0.3295595943927765, -0.25331658124923706, 0.03838096931576729, 1.0476552248001099, -0.19010882079601288, 0.2986795902252197, -0.9664064645767212, 1.7216105461120605, 0.04854540899395943, 0.30682337284088135, -0.8329529166221619, 0.015785537660121918, -0.47287631034851074, -0.2125675082206726, -1.6027798652648926, -0.9618031978607178, 1.3589116334915161, 1.2034952640533447, -1.7973461151123047, 0.21557053923606873, 1.8709909915924072, -1.0094960927963257, 1.1124796867370605, 0.3826877772808075, 0.9961821436882019, -0.9034715890884399, -2.2292399406433105, -1.9134902954101562, -0.6007419228553772, 0.27972277998924255, -0.41816744208335876, 0.5630418062210083, -1.131341576576233, -0.5997004508972168, 0.26869964599609375, -0.5647252202033997, 0.16955098509788513, -0.09878019988536835, -0.3560960590839386, -0.05318570137023926, -0.24052101373672485, 0.6571935415267944, -0.7371338605880737, -0.48880836367607117, -0.5840716361999512, -0.5980595350265503, -0.2297516018152237, 0.032808177173137665, -1.4163782596588135, -0.8084718585014343, 0.6249633431434631, 0.3239850401878357, 0.6727484464645386, 1.3891257047653198, -1.2307778596878052, -1.1035391092300415, -0.21795451641082764, -1.3125540018081665, 1.2674630880355835, -0.38569897413253784, 0.7251996397972107, -0.6911597847938538], [-0.4241904318332672, 1.7995268106460571, -1.0894811153411865, -0.5008487105369568, 0.32287687063217163, -1.4301860332489014, 1.1241896152496338, -0.04789057374000549, 1.107756495475769, -0.030949214473366737, -1.8495261669158936, -1.0592145919799805, 0.3548663258552551, 1.1195279359817505, 0.34013140201568604, 0.3398950397968292, 0.006923092994838953, 1.0955110788345337, -1.242714524269104, 1.3433375358581543, 1.510180115699768, 2.4791290760040283, 0.918481171131134, -0.09292095899581909, -0.7410224676132202, -0.7182196974754333, 0.9549736976623535, 1.00651216506958, 0.5233829021453857, 0.5764867067337036, 1.1672042608261108, 1.9668852090835571, 1.9860286712646484, -0.7439323663711548, 1.0497130155563354, -0.9116268754005432, -0.45025038719177246, 1.8175227642059326, 0.6583864688873291, -1.1307281255722046, -1.6367464065551758, -1.828099250793457, 2.0179057121276855, 0.38496264815330505, 1.2108595371246338, 0.47345244884490967, -0.9562491774559021, -0.6876091957092285, -0.03599380701780319, -1.7725921869277954, 0.5629463791847229, -0.2214096039533615, 0.5835486054420471, 0.34322357177734375, -0.7462255954742432, -1.3251394033432007, -0.03996261581778526, 0.5775235295295715, 0.33532819151878357, -0.011009518057107925, 0.276324987411499, -0.5660744905471802, 0.5411022305488586, 1.190487265586853, -0.7129532694816589, -0.22807887196540833, -1.8388569355010986, 0.11978752166032791, 0.5112389326095581, 0.057773396372795105, -0.0061811539344489574, -0.4791719615459442, -1.0284329652786255, 0.7614606618881226, 0.14843615889549255, -0.7954841256141663, -1.3861314058303833, -1.1745983362197876, 1.1694254875183105, 0.1087067723274231, 0.5639129281044006, -0.760526716709137, -0.2465570867061615, 0.2224908471107483, 0.22504456341266632, -0.8487432599067688, 0.31478211283683777, -0.32000532746315, -0.9079991579055786, 1.8006362915039062, -0.4320257604122162, -0.6072447896003723, -0.5985503196716309, 1.1344205141067505, -0.5094218850135803, 0.22630198299884796, -1.317912220954895, -1.0172408819198608, 0.19372710585594177, 0.6539745330810547, 0.22983548045158386, -1.2273330688476562, 0.2522120773792267, -0.4528566300868988, 0.33978140354156494, 2.0813658237457275, 1.326431393623352, -1.4950836896896362, -0.9959408640861511, -1.440261960029602, -0.6915403604507446, -0.4250953495502472, 1.2747987508773804, 0.03276927024126053, 1.7349509000778198, 0.8836238980293274, 0.27422115206718445, -1.2667276859283447, 0.7275974154472351, 0.795490562915802, 0.9595288038253784, -1.3353725671768188, -1.8993884325027466, -0.9101001024246216, -0.7253758311271667, 0.07072540372610092, 2.1865575313568115, -0.751387357711792, 1.4152379035949707], [-0.061250362545251846, -0.00031222269171848893, 1.2043702602386475, -1.9079982042312622, -1.2967103719711304, -0.26554715633392334, 0.7566856741905212, 0.5557910799980164, 1.1028492450714111, 1.3502768278121948, 0.16256025433540344, -1.3735848665237427, -0.49397704005241394, 1.1249982118606567, -1.5526323318481445, 0.45107513666152954, -0.42408230900764465, -1.3524574041366577, 0.8095097541809082, 0.06034541875123978, 0.8401684165000916, 0.2268054485321045, -0.8051064610481262, 0.51180499792099, 1.1592530012130737, -0.7548189163208008, 0.029226593673229218, -1.4311257600784302, 1.0739448070526123, -2.3293731212615967, -1.4271376132965088, 1.989995002746582, 1.209118366241455, 0.9929327368736267, -0.9867010116577148, 0.6824703812599182, 0.821085512638092, 1.7371935844421387, 1.38040030002594, 0.41863593459129333, -0.41526877880096436, 0.7124367952346802, 1.578143835067749, 0.7521040439605713, -0.59359210729599, 1.0609173774719238, 0.23632259666919708, 1.3411754369735718, 1.0388027429580688, 0.8228474855422974, 0.33443281054496765, -1.4887157678604126, -1.4268301725387573, -0.5616552829742432, 1.2481275796890259, 0.5497845411300659, 0.3414711654186249, -0.6171815395355225, -0.0010879887267947197, 0.7215652465820312, -1.1255383491516113, -1.3087115287780762, -0.8681846857070923, 1.3245415687561035, -0.36702439188957214, 0.6415917277336121, 0.8921681046485901, 0.41625678539276123, 1.847838044166565, -1.5193462371826172, -1.2303783893585205, 0.19484807550907135, 0.444170743227005, 0.044029511511325836, -0.3540067672729492, -1.350281834602356, 1.3112953901290894, -1.2420989274978638, 0.5612849593162537, 1.4020049571990967, 0.24987949430942535, -0.6696291565895081, -0.06098388135433197, -0.2132946252822876, 0.4642447829246521, -0.6270272135734558, -0.7855799198150635, 0.3802799582481384, 0.026144741103053093, 0.10291308909654617, 1.3432650566101074, 0.8709779381752014, -0.2401358187198639, -0.05774283409118652, 1.5386353731155396, 0.2801656723022461, -0.0544096976518631, 0.6940505504608154, 1.175696849822998, -0.5437014102935791, 0.978883683681488, -2.1264326572418213, 0.06050855293869972, -0.09043369442224503, -0.11628333479166031, -1.6735997200012207, 0.6803474426269531, -0.9582470655441284, 0.34299832582473755, 0.3840709626674652, -0.36143678426742554, -0.03290529176592827, -1.1925981044769287, -0.43422573804855347, 0.07689584791660309, -0.11004457622766495, -1.796770453453064, 0.741126298904419, -1.898290991783142, 0.19958168268203735, 1.5939737558364868, -2.2349801063537598, -1.7862564325332642, -0.6949779391288757, 0.6696755886077881, -0.9144829511642456, 2.0401906967163086, 1.140893578529358, 0.6559454202651978], [-0.8873851299285889, -1.1159849166870117, -0.4049213230609894, -0.18548044562339783, 0.4903157949447632, -0.8256794810295105, 0.42093634605407715, -0.9333643913269043, 0.8317917585372925, 0.8190104961395264, -0.9197942614555359, 1.0549466609954834, 2.3233373165130615, 0.7137248516082764, -0.16439862549304962, -0.4819640815258026, -1.7952736616134644, 0.8584949970245361, 1.2385882139205933, -0.5393912196159363, -0.06642334163188934, -0.5178864002227783, -0.0637294203042984, 0.6596156358718872, 1.3140937089920044, 1.1710611581802368, -0.4260428547859192, 0.09497029334306717, -0.7943940162658691, 0.7077187895774841, -0.47954797744750977, -0.7291970252990723, -0.7986477613449097, 0.12987083196640015, -0.6963410973548889, -0.11501042544841766, 0.13347099721431732, 0.3856985867023468, -1.8407588005065918, 1.0362468957901, 0.25909537076950073, 1.0030525922775269, -0.35748666524887085, 0.6363162994384766, -0.8562469482421875, -0.4186365306377411, -0.3791181147098541, 0.8443838953971863, -0.8984509706497192, 0.23481175303459167, 0.16906984150409698, -0.5946249961853027, 0.6292470097541809, -0.5494661927223206, -0.6731178164482117, 0.2556387484073639, 0.01988868974149227, 0.11668216437101364, 1.8465806245803833, 1.0613912343978882, 0.30657175183296204, -0.11634859442710876, 1.8327573537826538, 2.015864133834839, 0.43118149042129517, 1.2378172874450684, 0.09238491952419281, -0.6882931590080261, -0.875817060470581, 0.08030976355075836, -1.5009944438934326, 0.5861203670501709, 1.2773431539535522, -1.3158719539642334, -0.8987433910369873, -2.1261935234069824, 0.1697300523519516, 1.0240172147750854, 0.09045790135860443, -0.11374378204345703, 0.5427690744400024, -0.9690760970115662, -1.093563199043274, 2.3481221199035645, -2.0089187622070312, 1.1798934936523438, 0.238661527633667, 0.6708117723464966, -0.06505916267633438, 0.5662974119186401, -1.9145283699035645, 1.4535197019577026, 0.842282235622406, 1.1761248111724854, 1.3873364925384521, -2.1300816535949707, 0.36961883306503296, 1.591202735900879, 0.9173881411552429, -0.6474149823188782, -0.1928919553756714, 0.7117167711257935, -0.014839374460279942, -0.7162933945655823, -0.12066956609487534, 0.20872631669044495, 1.166433572769165, -0.06129363179206848, 0.869751513004303, -0.16411092877388, 1.319315791130066, 0.013749195262789726, 0.616106390953064, 0.0050428626127541065, 0.3600793480873108, 1.1286424398422241, 0.02258525975048542, 1.0254923105239868, -0.05936489626765251, 2.108213424682617, -0.7755808234214783, -1.1521503925323486, -0.7484914660453796, 0.19456946849822998, 1.2155898809432983, -1.2190972566604614, 2.371631145477295, -1.0687553882598877, -0.6788003444671631], [-0.2209755778312683, 0.28272125124931335, 1.0182559490203857, -1.281353235244751, 0.7581961750984192, 1.8713680505752563, -0.6769653558731079, -0.43845975399017334, 0.11075815558433533, 0.1065676212310791, -0.4678642749786377, -0.5132046341896057, -0.028788389638066292, 0.16952309012413025, 1.05326509475708, -1.0956165790557861, 0.7603968977928162, -1.4136934280395508, -0.8550627827644348, 0.877830982208252, 0.32753464579582214, 0.7180747389793396, 1.275131106376648, 0.4097936153411865, -0.16792035102844238, 1.5931981801986694, -0.7333388924598694, 3.1415774822235107, -1.972834825515747, -0.6634804010391235, 1.4857442378997803, -0.9071938991546631, 1.2555838823318481, 0.30787351727485657, 1.4766207933425903, 0.13632312417030334, -0.17466941475868225, 1.05615234375, 0.9957530498504639, 1.1064285039901733, -0.3273816406726837, -0.08789131790399551, 0.6464710831642151, 2.3495073318481445, -0.2211000919342041, -0.41119569540023804, 0.05769449844956398, 0.877337634563446, 1.063747525215149, 0.4274466633796692, -1.0670545101165771, 2.050232172012329, 1.4421229362487793, 1.8860430717468262, -2.5878946781158447, 1.2304846048355103, -1.1544667482376099, 0.5662916302680969, 0.5758516192436218, -0.37923040986061096, -1.1390204429626465, 0.7635226249694824, -1.2849103212356567, -1.9336050748825073, -0.8178243637084961, 0.25654590129852295, 0.4997473359107971, 0.18707403540611267, 0.5682855844497681, -0.03039899282157421, -0.312859445810318, 0.46186572313308716, 0.7301517128944397, -0.27923041582107544, 0.4691953659057617, 0.26142024993896484, 0.9599694609642029, 0.10760465264320374, -0.49835309386253357, 0.18822482228279114, -0.7295713424682617, -0.8486458659172058, -1.2722095251083374, -0.09467735886573792, 0.11889542639255524, 0.6848868727684021, 0.8570334911346436, -1.885911464691162, -0.9510190486907959, 0.7951529026031494, 1.4593658447265625, 0.05567389354109764, -0.6986300349235535, -0.1205778643488884, -0.9315480589866638, 1.5551327466964722, -0.1236412301659584, -0.39874809980392456, -0.6888078451156616, -0.7096830606460571, 0.8518232107162476, -1.7380424737930298, -0.45214352011680603, 0.8122897148132324, -0.4893220365047455, 1.308768391609192, 0.7393031120300293, 0.7218396067619324, 0.7865723371505737, -1.0598756074905396, 0.877085268497467, 0.45975589752197266, -1.0411715507507324, 1.7938729524612427, -0.49483755230903625, 0.3125152587890625, 0.5595899224281311, 0.37449073791503906, 2.073603391647339, -1.4256232976913452, 0.4071904122829437, -0.09802968800067902, 0.12535355985164642, 0.5097158551216125, -0.5800331234931946, 0.6008962988853455, -0.6132717728614807, -0.027162741869688034, -0.21138879656791687], [-1.2567706108093262, -1.2911921739578247, 0.45651987195014954, 0.04027941823005676, -1.0924577713012695, -0.8415191173553467, -0.20212669670581818, -0.7038184404373169, 1.1900519132614136, 0.6813679337501526, 0.11459322273731232, 0.4209461808204651, -0.7642660737037659, 0.47475484013557434, 2.163498640060425, -0.03096592240035534, -0.913460910320282, -0.9630578756332397, -0.3506143391132355, -0.0076989377848804, -0.32279378175735474, 0.033944886177778244, 0.8549885153770447, 0.09706113487482071, 0.41675466299057007, -0.6618464589118958, 1.6106445789337158, 0.6461571455001831, 0.28308767080307007, 0.6479923725128174, 0.5769582986831665, -0.5854494571685791, -1.4343267679214478, -1.083966851234436, -0.4924122393131256, -0.022448569536209106, -0.4879748523235321, 0.5410069823265076, 1.4893841743469238, 0.3562427759170532, 0.6934086084365845, 1.0258715152740479, -0.5371884703636169, -0.6621186137199402, -0.6674752831459045, -0.023665733635425568, 0.16625480353832245, -0.7925858497619629, -0.7768058180809021, -0.2251066416501999, 0.9360901117324829, -0.8065877556800842, -0.08675003796815872, -1.2498865127563477, 0.6704855561256409, -0.5419989824295044, -0.06148028373718262, -2.2227530479431152, -1.454066514968872, 1.043710470199585, 1.0456746816635132, -0.9444370865821838, -0.32219401001930237, 0.7210957407951355, 0.2602684497833252, 0.012389982119202614, 0.5140777826309204, 0.38894614577293396, -0.38768771290779114, -0.6826722025871277, 0.7623493075370789, 1.0765596628189087, -0.5253393054008484, 2.123115062713623, -0.32470497488975525, 0.5913786292076111, -0.3179418742656708, -0.5750037431716919, 0.294481098651886, 0.10953985899686813, -0.7851971387863159, -0.510840892791748, 0.06925520300865173, -0.7199303507804871, 0.9068882465362549, -2.4157826900482178, 1.1762313842773438, 1.6978082656860352, -1.1965469121932983, 1.9626851081848145, 2.092310667037964, -0.4746195673942566, -0.4454534947872162, -1.156315803527832, -0.14228488504886627, -0.10024113208055496, 0.43800902366638184, 0.020331421867012978, -1.4586750268936157, 1.6589609384536743, -0.20030730962753296, 0.2584587633609772, -1.4287066459655762, -0.5310956239700317, -1.0717744827270508, 1.2625924348831177, 1.4081766605377197, -0.17681248486042023, 1.0947277545928955, 2.484602212905884, -0.05104224011301994, 1.03191339969635, -0.6118627190589905, 1.9261853694915771, 0.31363239884376526, -0.16021688282489777, -1.178731918334961, -1.1393027305603027, 1.1943663358688354, 0.5534090399742126, 0.5297690033912659, -0.19255322217941284, 1.7120864391326904, 0.17663373053073883, -0.8265084624290466, 1.8166639804840088, -0.20496346056461334, 1.4727708101272583, 0.4510248601436615], [-0.5845958590507507, 0.06376570463180542, -0.8121333718299866, -0.5162490010261536, 1.2411381006240845, 0.023482423275709152, -0.3366466462612152, 0.8309011459350586, 1.3082751035690308, 0.17720548808574677, 0.1362750083208084, -1.2887641191482544, 0.4434658885002136, 1.2312617301940918, 1.171598196029663, -0.34425005316734314, 2.205080986022949, -0.7379617691040039, -1.6967029571533203, 0.9899834394454956, -1.8958444595336914, 0.08749359846115112, 0.5923755168914795, 0.5672825574874878, -1.0319123268127441, -0.9419155716896057, 0.2822122871875763, -0.7366225123405457, 0.7231084108352661, -0.43997135758399963, -0.4758131504058838, 0.721585750579834, -0.0826403871178627, -0.09004788100719452, 0.7076728940010071, 0.7431055903434753, 0.5842167735099792, 2.2321689128875732, -0.675006628036499, -0.21848683059215546, 2.226222515106201, -0.5286867618560791, -1.252813458442688, 0.3972918689250946, -1.3426498174667358, -0.4134122431278229, 1.635468602180481, 0.9583367109298706, 0.30744460225105286, 0.6512686014175415, 0.8905622959136963, -0.27027428150177, 1.9026896953582764, 0.5730394124984741, -0.5254412889480591, -1.4366333484649658, 1.0325963497161865, -1.4214396476745605, -0.07161165773868561, 0.9680248498916626, 0.22135381400585175, -0.035668935626745224, -0.22174634039402008, 0.6449210047721863, 0.41872158646583557, 1.6002187728881836, 1.0604283809661865, 1.8684855699539185, 1.6838812828063965, -0.8089016675949097, 1.8905854225158691, -1.7857956886291504, 0.080143503844738, -0.8813674449920654, 0.12842077016830444, -0.4758136570453644, 0.06108752638101578, 1.021239161491394, -3.126258134841919, 1.5735231637954712, -1.0251636505126953, -1.609269380569458, 0.3983977735042572, 1.1299296617507935, -0.7658177614212036, 0.6539157629013062, -1.2021918296813965, -0.09652559459209442, 0.23938718438148499, -0.18922901153564453, 0.046630240976810455, -0.4786154329776764, 2.0176143646240234, 0.40305471420288086, -1.6945316791534424, 1.3659100532531738, -0.7202991247177124, -0.6581088304519653, -0.013952336274087429, -1.240540862083435, 2.3506500720977783, -1.2028285264968872, 0.16790242493152618, 0.05405466631054878, 1.7371224164962769, -1.4332778453826904, 0.9037563800811768, 0.3765890300273895, -1.7675178050994873, 1.6893422603607178, -1.193871021270752, 0.31656694412231445, -0.7792440056800842, 1.2469865083694458, 0.2521064877510071, 1.2561349868774414, 1.1719634532928467, 0.9657138586044312, -2.390913486480713, -0.104000523686409, 1.2461836338043213, 0.9678112268447876, 0.3424251079559326, -0.31989505887031555, 0.06222320720553398, -0.117044597864151, 0.18866467475891113, -0.45794907212257385, 0.88128262758255], [-0.8055042624473572, -0.9253411889076233, 0.14405056834220886, 0.055162206292152405, 0.8793751001358032, -0.037912625819444656, 0.22740763425827026, 0.6418759226799011, 1.0091874599456787, -0.334127277135849, -0.860071063041687, 0.24619388580322266, 0.03244298696517944, 0.40756526589393616, -1.3561081886291504, 0.17315907776355743, 1.248170256614685, 1.5927212238311768, -2.5373694896698, -0.33168500661849976, 1.2648472785949707, 1.3312625885009766, 0.45239168405532837, -0.002003822010010481, 1.2813955545425415, -0.8638401031494141, -0.2746415138244629, -0.21309338510036469, -0.8948139548301697, 0.8522927165031433, -0.6137047410011292, -0.03073982521891594, 0.24782732129096985, 0.013209720142185688, 1.0731337070465088, 1.2671939134597778, 1.1264665126800537, 1.2335877418518066, -0.6438065767288208, -0.8167149424552917, -1.1954259872436523, -1.1440800428390503, -0.029674598947167397, -1.5157588720321655, 0.5683387517929077, 0.02867674082517624, -1.3306975364685059, 0.34424200654029846, 1.5372015237808228, 0.6062796711921692, -0.6110012531280518, -0.9975672364234924, 0.746192455291748, -0.08359566330909729, 1.4571431875228882, 0.7859423160552979, 0.1189466193318367, 1.8452975749969482, 0.10579238831996918, -0.7773826122283936, 0.19793321192264557, 0.3522675931453705, -1.1087027788162231, 0.8604506850242615, 1.7924787998199463, 0.6631218791007996, -1.8211601972579956, -1.2082276344299316, -0.3097859025001526, 0.958853542804718, -1.700676441192627, -0.6386924982070923, 1.023653268814087, -1.5511984825134277, 0.3900538980960846, -0.3559964895248413, 0.17477773129940033, 1.4247026443481445, 0.7954810261726379, 1.0615100860595703, 0.5985299944877625, 0.38210275769233704, -0.5304524898529053, -0.5487821698188782, -0.2484063357114792, -0.6027335524559021, -0.777932345867157, -0.057737357914447784, -0.47104692459106445, -0.25353965163230896, -1.2118518352508545, 0.09759081155061722, 0.4458392560482025, -1.8844410181045532, -0.5867710709571838, 0.4807088077068329, 0.799717128276825, 0.09822120517492294, 0.932429850101471, 0.9356411099433899, -0.14447131752967834, 0.3053184747695923, -0.21222208440303802, 0.08431819081306458, 1.198357343673706, -0.1202840507030487, -0.4028293192386627, 1.200933814048767, 0.7792781591415405, -0.7527163028717041, 2.0628106594085693, -0.25975489616394043, -0.4241029918193817, -1.1595133543014526, 0.28943389654159546, -0.03749828040599823, -0.22090984880924225, -0.9489680528640747, 2.6514294147491455, 0.3498733937740326, -0.5882657766342163, -2.6751275062561035, -1.6063470840454102, 1.4252885580062866, 0.266204297542572, -1.0996228456497192, -0.6259121894836426, -0.47967714071273804, 2.0529427528381348], [-0.6490521430969238, -1.0122150182724, 0.7274498343467712, 0.33728593587875366, -0.6396334767341614, -0.5264818072319031, 1.9436267614364624, -0.9510254263877869, -1.284174919128418, -0.31663933396339417, 0.7284477949142456, -1.6466275453567505, 0.341887891292572, -0.766187846660614, -0.1000618115067482, -1.2447118759155273, -0.23710544407367706, -0.7235497832298279, 0.5074869394302368, 0.48260778188705444, -0.7683147192001343, 0.8176577091217041, 2.3056657314300537, -0.9235620498657227, -0.8164103627204895, 1.048384189605713, -1.2302851676940918, 0.32969143986701965, 0.7085954546928406, -1.9635812044143677, 1.4957330226898193, 0.36910462379455566, 1.155206322669983, -0.13438592851161957, 1.6438630819320679, -0.663084089756012, -0.3334031403064728, 0.6943298578262329, 0.5317444801330566, 1.2528655529022217, -0.4284558892250061, -0.6861048936843872, 1.5692557096481323, 2.273066282272339, 0.23939457535743713, -0.004562622867524624, -1.3258211612701416, 0.5356715321540833, 0.3937331438064575, -0.6462826132774353, -1.0638244152069092, 0.5834189057350159, 1.007744550704956, -0.8537790775299072, 0.5825213193893433, -0.20775748789310455, -0.059295862913131714, -1.3990347385406494, -0.544684112071991, -1.6916626691818237, 1.2882159948349, -2.165691614151001, 1.5948258638381958, -0.3940545916557312, -1.0990344285964966, -0.28974080085754395, -0.8783276081085205, 0.8005976676940918, -2.0715508460998535, 0.2711978554725647, 0.5767610669136047, 1.2708910703659058, -2.006680488586426, -2.0160703659057617, 0.01662454754114151, -1.3140852451324463, -0.4527217447757721, -0.2511594593524933, 1.0621988773345947, 0.4674065411090851, -1.5376896858215332, 1.025961995124817, 1.6626930236816406, -0.8015464544296265, 0.9505103230476379, 1.4726638793945312, 0.615254819393158, 0.09113792330026627, -0.769745945930481, 1.4766128063201904, 1.49421226978302, -0.5579251646995544, 0.7844494581222534, -0.4511057138442993, 0.5270166993141174, -0.8960488438606262, 0.5282617211341858, 1.666850209236145, 0.2764562964439392, -0.1039600595831871, -0.5125412940979004, -0.03897814452648163, 0.8259070515632629, -1.8963029384613037, 1.1331769227981567, -1.0390880107879639, 0.9453796744346619, -0.8057132363319397, 0.5631718635559082, 0.7513399124145508, 0.1303967386484146, 0.6440132260322571, 1.54910147190094, 0.6881601810455322, -0.7507434487342834, 0.5154275298118591, 0.6350482702255249, 0.2614545226097107, 0.19815059006214142, 0.13462264835834503, 1.5145509243011475, -0.019024118781089783, -0.45764634013175964, 0.8196267485618591, -2.0207462310791016, -0.28143540024757385, 0.4188360273838043, 0.7188692688941956, 1.3609309196472168], [1.1879793405532837, 2.482858896255493, 0.05823850259184837, -0.01900295354425907, -0.5794223546981812, -0.0177918728441, -0.24862028658390045, 0.5310520529747009, 0.331855833530426, -1.2173234224319458, -1.558272361755371, 1.43193781375885, 1.7772433757781982, 0.2416587918996811, -0.12410817295312881, 0.47711384296417236, -2.8329052925109863, -0.7974847555160522, -1.2650058269500732, 0.6329209804534912, -0.08246105909347534, -0.3918313682079315, 0.6174213886260986, -2.1036345958709717, -0.27370309829711914, -1.7890554666519165, -0.2006102353334427, 0.9422481656074524, 0.4445778727531433, 1.883251428604126, 0.6341721415519714, -0.2255503237247467, -0.21930433809757233, 1.4979243278503418, -0.5242114067077637, 0.25157323479652405, -1.8469152450561523, -1.496358871459961, 0.47128236293792725, -0.190576434135437, 0.42966917157173157, -1.5676451921463013, -1.4884669780731201, -0.1805620640516281, 0.520806610584259, -0.9947191476821899, 0.9589418768882751, 0.7606232762336731, -0.3529706299304962, 0.5420737862586975, 0.13761501014232635, 1.7830588817596436, -1.0582146644592285, 0.9123300313949585, 0.7569169998168945, -0.20169448852539062, 0.6672564744949341, -0.2874045670032501, 0.35250356793403625, 2.2377443313598633, 2.6099441051483154, 0.29015877842903137, -0.014786573126912117, 1.1196976900100708, 0.18008100986480713, 0.6676497459411621, 0.448590487241745, 0.9326616525650024, -0.7749295830726624, 1.2092957496643066, 0.384860098361969, -0.1619294136762619, -0.6677943468093872, 1.097912311553955, 0.16726310551166534, 0.5994473695755005, 0.8253825306892395, -0.4781898856163025, 0.2159608155488968, 0.7330459356307983, 0.03359781950712204, -0.820643424987793, 0.46646416187286377, -2.1141626834869385, -1.9751354455947876, -0.3905152380466461, -0.9048328399658203, -0.4114997088909149, -0.7102469205856323, 1.386850357055664, 0.926235020160675, 0.5821762681007385, -1.0017427206039429, -1.2500019073486328, 1.2815945148468018, -1.467339038848877, 0.2649140954017639, -0.42324307560920715, 1.4564051628112793, 1.0143542289733887, 0.5478820204734802, -0.39726683497428894, 1.663520097732544, 0.36110013723373413, 0.8205766081809998, -0.5683816075325012, 1.4034144878387451, -0.5274062752723694, 0.5275745987892151, -1.9816269874572754, 1.5031726360321045, 0.11672947555780411, -0.5785568952560425, -0.006998459342867136, 1.0287282466888428, 0.09545228630304337, -0.7152523398399353, -0.046006519347429276, 1.555044412612915, 0.33677375316619873, -0.6263633966445923, -0.15423591434955597, -0.04470569267868996, 0.4006958603858948, -1.1842421293258667, -1.4391897916793823, -0.8061031699180603, 0.947905421257019, -0.2115338295698166], [-0.35190925002098083, -0.970844566822052, 0.5537707209587097, 0.03300445154309273, -1.8273576498031616, 0.6239321231842041, 0.30008599162101746, 0.454731285572052, -0.14917470514774323, 0.8768126368522644, -0.6617377400398254, -1.134310245513916, 0.3083634674549103, 0.6230540871620178, 1.4193326234817505, 0.4104224443435669, 1.4196768999099731, -0.5659493207931519, -0.14082206785678864, 0.42324358224868774, 0.25796061754226685, 0.5183600783348083, -0.7621662020683289, -0.4825384020805359, -0.8519315719604492, -0.6424576640129089, 1.6423479318618774, 1.297848105430603, 1.0883164405822754, 0.8461127281188965, 1.319567084312439, 0.16550078988075256, -2.9292776584625244, 0.7331193089485168, 0.39974990487098694, 0.4259309470653534, -0.09367257356643677, 1.2127271890640259, 1.219750165939331, -0.49368879199028015, 0.0057709054090082645, -0.9187310338020325, -1.1070674657821655, 1.5549137592315674, -0.9818037748336792, 0.975666880607605, -0.4820311367511749, -0.37898850440979004, -0.28327029943466187, -2.1682634353637695, 0.03777626156806946, 0.03531330078840256, 1.3136929273605347, 1.5694866180419922, -0.6670740842819214, 0.9154613018035889, 0.2697172462940216, -0.6490927338600159, 0.6035048365592957, -0.5903849005699158, -1.181799292564392, -0.5874820351600647, -0.26474305987358093, -1.5557825565338135, -0.664009690284729, -1.8185166120529175, -1.0317013263702393, -0.8164688944816589, 1.2955540418624878, -1.2023134231567383, -0.16295097768306732, -0.6657518148422241, 0.8342422246932983, 0.4038509428501129, -0.9195267558097839, -0.9608126282691956, -0.5490493178367615, 0.9493043422698975, -0.9642021656036377, -0.4652538597583771, -0.6925232410430908, -1.164555311203003, -0.9705036282539368, 0.5971847176551819, 0.3687271177768707, 1.0853080749511719, -0.011746713891625404, 0.1419648677110672, 1.8440712690353394, 0.1525431126356125, -0.0314667783677578, -0.6572834849357605, -0.5525693893432617, 0.3179605305194855, -0.5900149941444397, -1.1250239610671997, 0.2734391689300537, -1.6715493202209473, -0.9798470139503479, -0.16747574508190155, 0.43702706694602966, -1.2457090616226196, 1.0685828924179077, -0.864623486995697, -0.5449994802474976, -0.8387198448181152, 0.7963598370552063, -0.22970154881477356, -0.7337082624435425, 1.122841477394104, -0.06988248229026794, -2.077876567840576, 0.47163626551628113, -1.013980507850647, -0.18516890704631805, 0.10723177343606949, 0.434301495552063, 0.5406280159950256, -0.11347300559282303, 0.2531604766845703, 0.4774435758590698, -0.59121173620224, 1.5085158348083496, -0.5820874571800232, 0.7846558690071106, 0.03563784807920456, -1.6095257997512817, -0.05533144623041153, -0.5955818891525269], [0.5090866088867188, -0.07046306878328323, 0.8896266222000122, -0.2445267140865326, -1.3785533905029297, 1.3841458559036255, -1.3025776147842407, 0.6376747488975525, -0.010057269595563412, -1.76090669631958, 0.3660687506198883, -0.4349680244922638, -0.39488908648490906, -0.22777307033538818, -0.40362778306007385, -0.8267815709114075, 1.2315016984939575, 0.48939916491508484, -0.2726224660873413, -0.3253912627696991, 0.103500597178936, -0.7074159383773804, -0.5855011940002441, -0.5173088312149048, -0.7859243154525757, 2.240797758102417, 0.18272054195404053, 0.9886798858642578, 0.7573496699333191, -0.03345869854092598, 0.7575240135192871, 0.9911602735519409, -0.025805732235312462, -1.711023211479187, 0.24058789014816284, 1.5672188997268677, -0.3882138133049011, 0.5816899538040161, -0.3810310661792755, -1.057176113128662, -1.0303970575332642, -0.20184367895126343, -0.5240033864974976, 0.18818612396717072, -0.7810893058776855, -0.23385295271873474, -0.9310709238052368, -0.8769579529762268, 0.12282520532608032, 1.3711057901382446, -0.06891719251871109, 0.22068163752555847, -0.9742894172668457, 0.3132956027984619, 0.02552323043346405, -0.4591986835002899, 0.8807330131530762, -0.33354878425598145, -2.0285847187042236, -0.1940840780735016, -1.1627265214920044, -0.49792683124542236, 1.1184262037277222, 0.7720325589179993, -0.3703504502773285, 0.522218644618988, -0.3679710328578949, -0.6386885643005371, -0.6145920753479004, 0.3513231873512268, -0.012951625511050224, -1.4028245210647583, -0.5073478817939758, -0.6387467980384827, -0.9205647110939026, 0.94861900806427, 0.08964183181524277, 1.4232205152511597, 0.13420423865318298, -0.2753824293613434, -1.4070771932601929, 0.45829910039901733, 1.2969204187393188, -1.1863499879837036, -1.2468228340148926, -1.072280764579773, -0.97648686170578, 0.26611271500587463, -0.9909047484397888, 0.034040667116642, 0.45880255103111267, -0.01903114654123783, -0.954004168510437, -1.1337947845458984, -0.2778998613357544, 1.7926616668701172, -0.30800461769104004, -0.7417407035827637, 0.5197214484214783, -0.9876644015312195, 2.144470453262329, -0.6905041933059692, 0.37952500581741333, -1.0898085832595825, 1.219433069229126, -0.45221373438835144, -0.5871190428733826, 0.7740477919578552, 1.7391809225082397, -0.87796550989151, -1.1997495889663696, 0.6401516199111938, -0.32063883543014526, 1.1961064338684082, 1.451479434967041, -0.4894008934497833, -0.43403521180152893, -1.072906732559204, 2.112718105316162, 0.033341869711875916, -0.12620219588279724, -1.2334212064743042, -2.00708270072937, 1.409446358680725, 0.6413940787315369, 0.6651343703269958, -0.4157656133174896, -1.1939698457717896, -1.684454321861267], [0.6540116667747498, 0.49950072169303894, 0.47413966059684753, 0.3258797526359558, -0.6794211864471436, 0.7523066997528076, 1.148330807685852, -0.244155153632164, -0.13382765650749207, 0.2865014970302582, -1.0483845472335815, 0.153506338596344, 0.5181195735931396, 0.5401789546012878, 1.9340565204620361, 1.9590452909469604, -1.4648414850234985, -2.472212314605713, 1.3515557050704956, -1.4647005796432495, -2.502119302749634, 0.31228142976760864, 0.5290245413780212, -0.36750632524490356, -0.9036956429481506, -1.0019789934158325, -0.3634425401687622, -1.4970622062683105, 0.01278578583151102, -1.4622873067855835, 0.5529488921165466, -1.0497902631759644, 0.6781790256500244, -0.6538715958595276, 0.8086866140365601, -1.9014724493026733, -0.8928542137145996, 1.010481357574463, 2.0214340686798096, 0.21109622716903687, -0.4393574595451355, -2.2254796028137207, 0.3412173390388489, -0.9284465312957764, -2.2995569705963135, -0.11973970383405685, 0.46251770853996277, 1.4974422454833984, 0.4725962281227112, 0.31451958417892456, 0.54522305727005, 0.5362213253974915, 0.9201122522354126, 0.07502556592226028, 0.6222901344299316, -0.30088335275650024, 0.22568391263484955, 0.5616970062255859, 0.8178541660308838, 0.22515009343624115, -0.7308365702629089, 0.8147944211959839, 2.401859998703003, -3.7872493267059326, 0.1789114624261856, 0.14378884434700012, 0.8243803977966309, 0.6855810284614563, 0.7964419722557068, 2.3840596675872803, -1.0769679546356201, -1.5237877368927002, 0.46804407238960266, -1.1187504529953003, 0.19444268941879272, -0.6653395891189575, 0.4270074665546417, 2.125530242919922, -0.6462283134460449, 0.7653266191482544, -1.0859473943710327, -1.1111900806427002, -2.1638853549957275, 1.1769267320632935, -1.540132761001587, -0.9160013198852539, -0.3632033169269562, -1.0227622985839844, 0.3324817419052124, 0.5641822814941406, -0.4782990515232086, 0.40504342317581177, 0.05241621285676956, -1.3805078268051147, 0.9413810968399048, 0.6864365339279175, 0.10859551280736923, 0.5932750105857849, -0.3175541162490845, 3.003951072692871, -0.1303231567144394, 0.6330956220626831, -0.09259750694036484, -0.8639442324638367, -0.2247234731912613, 0.6684719324111938, 0.6720816493034363, 0.20560277998447418, -0.23024193942546844, -0.47962048649787903, 0.26711586117744446, 0.6873493790626526, -0.7360944747924805, -1.331516146659851, -0.18383394181728363, -0.49341362714767456, -0.9126182198524475, 0.3346494734287262, -1.2824193239212036, 0.8678140044212341, -0.46260523796081543, 1.2255651950836182, 0.3731326162815094, 0.3362545669078827, 0.5730296969413757, 1.6500648260116577, -0.7522729635238647, -0.9533972144126892, -1.0602151155471802], [-0.16866008937358856, -0.5873118042945862, 0.7270501255989075, 1.1287648677825928, -1.1194010972976685, 0.8914995193481445, -0.6483685970306396, 1.220588207244873, -0.11053825169801712, -1.044325590133667, -1.0288598537445068, -0.21751217544078827, 0.844666600227356, -0.8405779004096985, -1.6551958322525024, -1.3391703367233276, -0.0791965201497078, 0.07508905977010727, 0.2767314910888672, -0.616248369216919, 0.2284049242734909, -1.076367974281311, -0.6039701700210571, 0.06919166445732117, 0.695146381855011, -0.5378051400184631, 0.18657448887825012, 0.5792718529701233, 1.4985088109970093, -1.4371843338012695, 0.8193881511688232, 0.8684723377227783, -0.27476435899734497, 0.07187346369028091, 0.29541274905204773, -0.4489347040653229, -1.3059446811676025, 1.0020999908447266, -0.7034428119659424, -0.3328351676464081, -1.8970963954925537, -0.2148181051015854, -1.7561886310577393, -0.35720542073249817, -0.2507854104042053, -0.17458494007587433, 0.14842963218688965, 2.0921735763549805, -0.09622061997652054, -0.18121522665023804, -0.2609681189060211, 0.1967972368001938, -0.1903715282678604, 0.8279128670692444, -0.5253182649612427, -0.188510924577713, -1.5095959901809692, -0.8952229022979736, -1.3121713399887085, 0.9006542563438416, -1.6082090139389038, 0.36913231015205383, 0.09986954182386398, 0.19043570756912231, 1.3984845876693726, -0.9228907823562622, -0.39708012342453003, 0.7614278793334961, 0.38032492995262146, -0.16955623030662537, 1.6977341175079346, -0.1892261654138565, 0.3958132266998291, -0.4642356336116791, -1.2631561756134033, -0.30835381150245667, -0.07700704038143158, -1.9839271306991577, 0.39224717020988464, 1.1219719648361206, 0.0008428450091741979, 1.1548924446105957, -2.0764846801757812, -2.592482566833496, 0.1864989846944809, -0.5263240933418274, -1.3332217931747437, 0.02473273128271103, 1.482356071472168, -0.1185414269566536, 2.1343860626220703, -0.1793653815984726, 1.1084157228469849, -0.8756247758865356, -0.1909564882516861, -0.4957205057144165, -0.3868864178657532, -2.5523521900177, 0.0546862930059433, 2.0476901531219482, 0.4816225469112396, -0.6311495900154114, 0.3110367953777313, 0.533980667591095, -0.4988721013069153, 2.11072039604187, -1.1002368927001953, 0.4343705475330353, 0.8231967091560364, 0.6975817680358887, 0.4892352521419525, 0.17387443780899048, 0.7911654114723206, 0.5243812799453735, 0.19971203804016113, -0.025839583948254585, 0.5870338082313538, 0.7819327712059021, 0.1957409828901291, -1.3724517822265625, 0.24649900197982788, 0.6165051460266113, 1.355518102645874, 0.39132314920425415, 0.018872670829296112, 1.0330688953399658, -1.6018890142440796, 0.12434499710798264, -0.037584271281957626], [-0.5381993055343628, 1.2076654434204102, 0.9580214619636536, 0.6976872086524963, -0.3967798948287964, 0.10693039000034332, 1.44889497756958, 0.06326953321695328, 1.631730318069458, -0.35711222887039185, -0.8534632325172424, -0.6313303709030151, -0.2365814596414566, -0.41478395462036133, 0.10797324776649475, -2.2821803092956543, 0.5216754078865051, 0.7323629260063171, 1.0432415008544922, 1.192548394203186, 1.083192229270935, 0.8407494425773621, 0.4594784379005432, -1.5024627447128296, 0.4314030408859253, -0.3906707763671875, -0.074813611805439, 0.9807438254356384, 0.3402540981769562, -0.3538053035736084, 1.4140821695327759, 1.4342830181121826, -1.0237727165222168, -1.5487589836120605, -1.7930376529693604, 0.9666902422904968, 0.9336341023445129, 1.0997554063796997, 1.1100741624832153, 0.6520915627479553, 0.26206326484680176, -0.057696882635354996, 1.4702285528182983, -2.5067739486694336, -1.450575351715088, 0.3735061585903168, 0.7923701405525208, -0.8254261016845703, -1.02900230884552, -0.6537913084030151, 0.12948380410671234, 1.5510965585708618, 0.5212284326553345, 1.728938341140747, -2.0522897243499756, -1.806649923324585, -1.282982587814331, -0.15805043280124664, 1.7465230226516724, -0.11933927983045578, 1.2801806926727295, 0.4347962737083435, -0.7165428400039673, -0.005864893551915884, 0.8344096541404724, 0.42405936121940613, -0.17430204153060913, 0.2186541110277176, -0.32263386249542236, -1.8223153352737427, 1.0264817476272583, 2.1155905723571777, -1.117742657661438, -0.41769924759864807, 0.1698705554008484, -0.5531067848205566, -0.9929140210151672, 0.508744478225708, -0.5181105136871338, 1.0325207710266113, -1.0554972887039185, -1.215946078300476, 0.38909465074539185, -1.5492788553237915, -0.6664038300514221, 1.4028329849243164, -0.06985551118850708, 0.3661201596260071, -0.3848278522491455, -1.7755743265151978, 0.9141215085983276, -0.0088544562458992, -0.49875128269195557, 0.2884494662284851, 0.47786760330200195, 1.625036597251892, -0.8083258867263794, 1.6908634901046753, -0.05561697483062744, 0.8640431761741638, -0.5870857834815979, 0.3504256010055542, 0.022136367857456207, 1.5301710367202759, 2.17677640914917, -1.2963954210281372, -0.17912323772907257, -0.31809505820274353, -0.657403826713562, -0.29472529888153076, -1.4534860849380493, -1.5739197731018066, 0.1972331404685974, 0.5084431767463684, 0.6944550275802612, -1.2923485040664673, -0.09847965836524963, 0.223984956741333, 0.6836017370223999, -1.6152067184448242, 1.7308049201965332, -0.974227786064148, 1.9607611894607544, 0.7402473092079163, 1.8178128004074097, -0.8729443550109863, -0.17301756143569946, 0.4540978968143463, -0.024467939510941505], [-1.0702415704727173, -2.165034770965576, -0.14766904711723328, 0.11482395976781845, -0.8586848974227905, -0.3449028730392456, 0.8016192317008972, 1.0533725023269653, -0.10702364146709442, -1.6350828409194946, 0.6803876757621765, 0.04001970216631889, -0.5271412134170532, 1.5816570520401, 1.0104416608810425, -0.41723188757896423, 1.4931621551513672, -1.1108957529067993, 0.26402387022972107, 1.0715817213058472, -0.3701239824295044, -0.9294776916503906, -0.7825176119804382, 0.23123760521411896, -1.696736454963684, -0.509243905544281, 0.03605014458298683, -1.173490047454834, -0.6293582916259766, -1.053812861442566, -0.7766609191894531, 0.4124859869480133, 1.3321595191955566, 1.4864940643310547, 0.10760922729969025, 0.04766980558633804, -0.5599132180213928, -1.8328129053115845, -1.0057541131973267, 0.10876238346099854, -0.474386990070343, 0.4212208092212677, 0.11235792934894562, -0.1651371717453003, -1.0428659915924072, -0.18448178470134735, -0.3814852833747864, -1.5411747694015503, 1.1496288776397705, -1.0432733297348022, 2.134199380874634, -0.5675820708274841, 0.04606151953339577, -1.0394529104232788, -1.2318955659866333, -0.5525872707366943, 0.14196479320526123, 1.4557547569274902, -0.3318522870540619, -0.6600456833839417, -2.0926427841186523, 0.4845545291900635, 0.1293390542268753, 0.5257940888404846, 1.0956968069076538, 0.2569426894187927, -1.615020751953125, 1.3402624130249023, -1.7246755361557007, -0.47940948605537415, -0.702290415763855, -0.25034311413764954, -1.5624046325683594, -0.022888053208589554, 0.7128849625587463, -1.0217652320861816, 1.5096783638000488, 1.2563623189926147, 1.7808104753494263, -0.4877835214138031, -1.953352689743042, -0.36508089303970337, 0.5159966349601746, 0.884195864200592, -0.8887333869934082, -0.7313485741615295, 1.2074694633483887, 2.4897313117980957, -2.792091131210327, 1.8942551612854004, -0.907367467880249, -1.9135780334472656, -1.2393547296524048, -1.283963680267334, -0.6224777698516846, 0.2658815383911133, 1.498786449432373, -0.030789818614721298, -0.019297083839774132, 0.3511745035648346, -2.045090675354004, 1.4058456420898438, -2.128295660018921, -1.227919578552246, -1.7480741739273071, -0.11750443279743195, -0.17259730398654938, -1.499648928642273, 0.33326682448387146, 0.5366498827934265, 0.424476683139801, -0.7038580179214478, 1.031734585762024, 0.5324793457984924, -0.7313650250434875, -0.8953310251235962, -2.534008264541626, -0.4690399765968323, -0.010489248670637608, 0.14335867762565613, -0.39804983139038086, -1.6315295696258545, -1.7602689266204834, -0.24101023375988007, -1.2389390468597412, 0.5082298517227173, 2.205130100250244, 0.8559122085571289, -0.04146347567439079], [-0.7447707056999207, 0.6629559993743896, 2.3901147842407227, 1.2951291799545288, -1.9275449514389038, 0.08891476690769196, -1.0113005638122559, 3.3699448108673096, -0.11505137383937836, -0.8607841730117798, 0.009184923954308033, 0.22402676939964294, 1.034395694732666, -1.9330366849899292, 2.6206560134887695, 0.7459045052528381, 1.1649338006973267, -1.2028392553329468, -1.519268274307251, -0.04367528110742569, -1.0955172777175903, -1.6908597946166992, 0.5576885938644409, 0.8455348610877991, -0.7081466317176819, 0.5456563830375671, 1.5662950277328491, 1.140863299369812, 2.2519707679748535, -0.38763320446014404, -0.6230430603027344, 0.9088069796562195, -1.2902251482009888, 1.606036901473999, -0.8949252367019653, -0.9240835309028625, -0.14003150165081024, 1.3299161195755005, 1.9017910957336426, 0.6866398453712463, -1.0249338150024414, -0.544279932975769, 1.923624038696289, -2.295789957046509, 1.5141856670379639, 0.1712665855884552, 2.2474286556243896, 0.4908972382545471, -1.1007931232452393, -0.1471101939678192, -1.3564274311065674, -0.9762713313102722, 0.001999938627704978, -1.0574569702148438, -0.4307476878166199, -0.03917081654071808, -0.3146370053291321, 0.6724135279655457, 0.4353807270526886, 0.5053978562355042, 0.43514448404312134, 0.10852959752082825, -0.8142309188842773, -0.6669005751609802, 0.5797025561332703, -2.7170441150665283, -1.0891926288604736, 0.49743571877479553, -0.1075073629617691, -0.3228191137313843, -0.6580361723899841, 0.786706805229187, -0.7675832509994507, 0.25910961627960205, -0.1597285121679306, 0.8095967173576355, -0.5762428045272827, 0.15893223881721497, -2.212803840637207, 1.2833781242370605, 1.210260272026062, 0.7786480784416199, 0.20720691978931427, -0.42863038182258606, -0.4616416394710541, -0.8072446584701538, 0.6092949509620667, -1.1685012578964233, 1.088003396987915, 0.8102961778640747, -0.09357930719852448, -0.17924852669239044, 0.4645954668521881, 0.4462225139141083, 0.0685141310095787, 0.3225118815898895, -1.110912561416626, 0.524749219417572, 0.15673431754112244, 0.25882312655448914, 0.6873165965080261, -0.22548224031925201, -0.3638414144515991, -1.1518062353134155, 0.6961563229560852, 1.6249628067016602, -0.4118264615535736, 1.159788966178894, -0.4362546503543854, 1.1617214679718018, -0.9749213457107544, -1.808670163154602, -0.7887308597564697, -0.911699652671814, -0.4172717034816742, 3.324814558029175, 1.7851159572601318, -0.2696476876735687, 0.7594499588012695, 1.195448875427246, 0.8083840012550354, -0.020963100716471672, 0.3974332809448242, 1.8938060998916626, -1.278091549873352, 0.9264034628868103, 0.01735849119722843, 1.1346524953842163, -0.0769490897655487], [-1.0155147314071655, -1.1039053201675415, -1.1305431127548218, 0.32222768664360046, 0.09093619883060455, -1.739983081817627, -1.1043583154678345, -0.2933816909790039, -0.6883639693260193, -1.0164945125579834, 0.11990698426961899, 0.1568831205368042, 0.1352526694536209, 0.9039007425308228, 1.5356976985931396, -1.0372017621994019, 0.6374462246894836, 0.5202069878578186, -0.87519770860672, 1.0249876976013184, -0.29067859053611755, 0.5419473052024841, -1.2086209058761597, 0.8654220700263977, -0.04506668448448181, 1.1541866064071655, 1.4099211692810059, -0.2866235673427582, 1.5145273208618164, 0.22345027327537537, 1.7259966135025024, -0.5275905728340149, 0.7482426762580872, 1.5488097667694092, -0.6286259889602661, -0.158066526055336, 1.4947514533996582, -0.4056752026081085, -2.160454034805298, -0.22414548695087433, 0.5494521856307983, -0.7077783942222595, 0.7911996841430664, -1.5983847379684448, -0.9517176747322083, 0.445485919713974, -1.013196587562561, 0.18082326650619507, 0.13620206713676453, -1.48941969871521, -1.2310670614242554, -0.49685755372047424, 0.9736381769180298, 1.4930782318115234, 0.12719622254371643, 0.593076229095459, -1.9015079736709595, 0.4213903844356537, 0.34945082664489746, -0.8678873181343079, -0.2747040390968323, 2.222632884979248, 1.1105566024780273, -2.172215223312378, -0.7019659280776978, 0.4725618064403534, -0.5522204041481018, 0.33536991477012634, -1.1091017723083496, -1.2330442667007446, -0.6180853247642517, -2.2369956970214844, 0.0423361100256443, -0.21904614567756653, 0.03957339748740196, -0.6814483404159546, 0.0010129149304702878, 2.17942214012146, -0.1552320122718811, -0.644218385219574, 0.2563537657260895, -0.9409252405166626, 1.1614238023757935, -0.8105762600898743, 0.3723958134651184, -0.9297303557395935, -0.34588468074798584, -1.55447518825531, 1.3543037176132202, -0.43103593587875366, 0.36456406116485596, -0.74106365442276, 0.3964853882789612, 0.43379685282707214, -1.2729146480560303, -1.9070724248886108, 0.403534859418869, -0.5344671607017517, -0.24460506439208984, 0.32626068592071533, 1.4273008108139038, 1.3978854417800903, -0.7467724680900574, -0.31028375029563904, -0.25011515617370605, 1.0803884267807007, 0.10533007234334946, -0.5682424902915955, 0.02812706120312214, 0.14721666276454926, -0.3849582374095917, 1.4001049995422363, -0.5623919367790222, 0.4941985607147217, -0.18087387084960938, -0.08636978268623352, -2.7060375213623047, -0.8041806221008301, 1.6103057861328125, -1.3209655284881592, -0.9934794306755066, -0.5603187084197998, 0.4677777588367462, -0.5733861327171326, -0.03396003693342209, -0.11548157781362534, 0.6904057264328003, -1.0731000900268555, -0.056065164506435394], [0.9168365001678467, -0.4717910587787628, 0.8656002283096313, -0.2573249936103821, -0.01593797281384468, 0.8007144331932068, -1.621468424797058, 1.2539231777191162, 0.7761818766593933, -1.0474873781204224, 1.7948194742202759, -1.0851646661758423, 0.42558327317237854, 0.38768935203552246, 0.6828508377075195, 1.2725735902786255, 0.6015653610229492, -1.6177479028701782, -0.9193247556686401, -0.11349988728761673, -0.7317451238632202, -1.3173972368240356, 0.30903249979019165, 0.8349327445030212, -1.4300751686096191, -0.4020879566669464, 0.1280515342950821, 0.3644373416900635, -1.4425705671310425, -0.4234575927257538, -1.0694881677627563, 1.6794962882995605, -1.1944957971572876, 1.06068754196167, 0.48981934785842896, 0.17132815718650818, 0.4517000913619995, -0.34947121143341064, 0.9453786015510559, -0.5369939804077148, -0.9383063316345215, -0.4273033142089844, -2.9380650520324707, -0.7930482625961304, 0.8620955944061279, 1.698995590209961, 0.401756227016449, -0.35641050338745117, -0.7944805026054382, 1.028926134109497, -1.1116681098937988, -0.8701356649398804, 1.7667173147201538, -0.5974394083023071, 0.3814872205257416, -0.43468794226646423, 0.5835630893707275, 0.28775399923324585, 0.9756292104721069, -1.0435912609100342, -0.4710327386856079, 0.3087742328643799, 0.99824458360672, -0.6581090092658997, 0.3164746165275574, -0.13370734453201294, 0.5825544595718384, 0.36042913794517517, 1.314621925354004, 0.1102604866027832, 0.3878694474697113, 1.115598201751709, -0.15539288520812988, -1.192186713218689, -1.2931114435195923, 0.22979550063610077, 0.6876861453056335, -0.08128006011247635, -1.058024525642395, 0.6128029227256775, 0.07792027294635773, 1.6605526208877563, 0.36060830950737, 0.5735713243484497, 0.5838479399681091, 0.5547904372215271, -0.7478466033935547, -1.8932901620864868, 0.2304212599992752, 1.2379765510559082, 1.9978880882263184, -1.4903578758239746, -0.805740237236023, 0.8294093012809753, 0.8727706670761108, -0.4114452600479126, -0.9604434370994568, 1.2208675146102905, 0.22760845720767975, -0.9578116536140442, 1.1125993728637695, -0.5929110050201416, 1.105699062347412, -1.0376975536346436, -0.8532770872116089, -1.0583423376083374, -2.1187613010406494, 1.8379414081573486, 0.4960052967071533, -0.7871956825256348, -0.39850398898124695, -0.3823945224285126, -0.7029478549957275, 0.24660314619541168, 0.18491461873054504, 0.17025987803936005, 0.19859468936920166, -1.3240625858306885, 0.4387345314025879, -0.01803216151893139, -1.2225741147994995, 0.8218461871147156, -1.7044974565505981, 1.396481990814209, 1.0312901735305786, 0.9906627535820007, -0.7717106938362122, -0.7311105132102966, -0.6137281656265259], [0.8527066707611084, 1.329033374786377, -0.2363489717245102, 0.40356940031051636, -0.3689194619655609, -0.5845078229904175, -0.4633271396160126, -1.0273449420928955, -2.055812120437622, 0.052872512489557266, 0.05981079116463661, -0.8457826972007751, 1.2819551229476929, -0.9303337931632996, 1.5171843767166138, -0.5717955231666565, -0.42944827675819397, -0.8722167611122131, 1.4194049835205078, 0.06597667187452316, 0.6658961176872253, -1.2881922721862793, -2.2566211223602295, 1.1869338750839233, 0.7401421070098877, -0.3456276059150696, -1.4065603017807007, -2.071669816970825, 1.3341106176376343, -0.5056436657905579, 0.8036941885948181, -1.2281004190444946, -1.3001633882522583, -0.4707750082015991, -0.8553887605667114, 0.583081066608429, 0.20422443747520447, -0.4783289432525635, -0.26810842752456665, 0.9997398853302002, 0.6755056381225586, -1.4035969972610474, 1.7264766693115234, -0.06406266242265701, -0.921610951423645, -0.3150731325149536, -0.14595216512680054, -0.9997237324714661, -0.5854511260986328, -0.26830530166625977, 0.34694841504096985, -0.0845731720328331, -0.9721975922584534, -0.18224874138832092, 0.5624686479568481, 1.0085268020629883, 0.8153899908065796, 1.3810428380966187, 2.373749017715454, -1.0091267824172974, -0.555780827999115, 1.255165457725525, 0.3358963429927826, 0.12168411165475845, -0.9553197622299194, -0.6274566054344177, -0.3317912817001343, -1.752223014831543, 1.4056154489517212, 0.34212833642959595, -0.8027080297470093, -0.33122479915618896, -0.6412739157676697, -0.1157878041267395, -0.9046705365180969, -0.11359279602766037, -2.1206843852996826, -0.6993198990821838, 0.9630761742591858, -0.25620442628860474, 0.7599398493766785, -0.6813860535621643, -0.22392666339874268, 0.966046929359436, -1.1218416690826416, -0.8039957284927368, 0.647908627986908, 0.10588480532169342, -1.9671080112457275, 0.9354184865951538, 1.607858657836914, -0.90672767162323, 0.458774596452713, 0.09772967547178268, -0.2994265556335449, 0.5601709485054016, 0.677177369594574, 0.4469970762729645, 0.6879503130912781, -0.8889038562774658, -0.23390302062034607, -0.7124249339103699, -0.53767329454422, 0.7019293308258057, 0.3528568148612976, 0.3288605809211731, 0.733656108379364, -0.8471024036407471, 0.7803674936294556, -1.5403140783309937, -0.30958443880081177, 0.25794264674186707, 0.3859865963459015, 0.2584967017173767, -0.4224262833595276, 2.5258171558380127, -0.8083710074424744, -0.47567129135131836, -0.7211518883705139, 0.4185922145843506, 1.5558255910873413, 0.758225679397583, 0.16612844169139862, 0.052434779703617096, 1.9483861923217773, -0.6981716752052307, 1.0119175910949707, 0.27347683906555176, -0.5830192565917969], [-0.8782836198806763, -1.6385364532470703, 0.4221177101135254, 0.4944693446159363, -1.7962846755981445, -1.3489012718200684, -1.7008721828460693, -2.046104669570923, -0.7351606488227844, 0.7306078672409058, 0.6544649600982666, 0.4399168789386749, -1.567373514175415, 0.6740369200706482, 0.9517868161201477, -0.29942458868026733, 0.36423543095588684, -0.7796836495399475, -0.8236585259437561, -0.276467502117157, -2.1733624935150146, -0.5375253558158875, 0.08639705181121826, 0.7050252556800842, 0.7536668181419373, 0.4530145525932312, -2.0904507637023926, -0.12178811430931091, 0.2676055133342743, 0.730871319770813, -0.33477550745010376, -0.3050994873046875, -0.3058760166168213, 1.5430225133895874, -1.046112298965454, 0.7193388342857361, -1.3503981828689575, -0.4457491934299469, 0.41066649556159973, 0.7246021032333374, 3.087458372116089, -0.09386328607797623, -0.7630611658096313, -1.9133236408233643, -0.8691479563713074, -1.8055245876312256, -0.08739957958459854, -2.0052437782287598, -0.6089116930961609, -0.7378076910972595, -1.5047056674957275, -0.40678972005844116, -0.19571736454963684, 1.4146037101745605, 0.386144757270813, 0.12473709136247635, -2.6649153232574463, -0.09342361241579056, -2.1914758682250977, -2.026057481765747, 1.5087395906448364, -0.47825902700424194, -0.2269604653120041, -0.05450556054711342, 2.9806947708129883, 0.16230334341526031, 0.024249745532870293, 0.6568812727928162, 0.35960161685943604, -1.6543205976486206, 0.09654425829648972, 0.2808232605457306, 1.1579304933547974, 0.43791884183883667, 0.7471166849136353, -1.1584479808807373, 1.0104061365127563, 2.211787700653076, -0.44676122069358826, -0.0513208843767643, -0.21223728358745575, -0.12290258705615997, -0.725736677646637, 0.06774115562438965, -1.2790277004241943, 0.06295276433229446, 0.18246716260910034, 0.908358097076416, 1.0146478414535522, -0.024590300396084785, 0.6143462657928467, -2.3194808959960938, 0.28003647923469543, 1.317602276802063, -0.941532552242279, -0.23867563903331757, 0.4809553027153015, 1.2282074689865112, -1.4327319860458374, 1.5550594329833984, 0.07564165443181992, -0.6900058388710022, -1.2490582466125488, -0.10260505229234695, 1.8709925413131714, 1.0723669528961182, -0.3685378432273865, 0.5801522135734558, -0.04106727987527847, 0.686842143535614, -2.318150758743286, -0.512005090713501, 0.7337931394577026, 0.33448079228401184, -0.9016632437705994, 0.5834574103355408, 0.4065135419368744, 0.6843042969703674, -0.425968736410141, -0.6771895885467529, 0.1065479964017868, -1.223619818687439, -0.8717881441116333, 0.6909042596817017, -1.6023950576782227, 0.3635329604148865, 0.2832747995853424, 0.6695286631584167, -1.4936234951019287], [-0.6547002792358398, 0.2567518949508667, 0.09588008373975754, 0.1709318459033966, 0.6677016019821167, 1.4841588735580444, 0.45770424604415894, 0.638110876083374, 1.3494594097137451, -0.3522021770477295, 0.9503058791160583, -0.702724814414978, 0.1615312546491623, 0.25083720684051514, -0.5102740526199341, 0.10973034799098969, -0.7788817286491394, -2.8070220947265625, 0.35072508454322815, -0.25107118487358093, -0.39936068654060364, 0.3367293179035187, 0.1374681144952774, -1.0073566436767578, 1.6945704221725464, 0.4522049129009247, 2.43381929397583, -0.7219244837760925, -1.756434679031372, 0.902125895023346, 0.6839551329612732, -0.02825228124856949, 0.30787941813468933, 1.3186012506484985, -0.6521596908569336, -0.47070327401161194, 0.04136725515127182, 0.4624192416667938, -0.3331153392791748, 1.2887217998504639, 1.752515196800232, -0.3646056354045868, -0.584415078163147, -0.1648794263601303, 0.38259902596473694, 1.8004240989685059, -0.2685043513774872, 1.2356528043746948, 2.160068988800049, 0.7040571570396423, -1.578066110610962, 0.3737718462944031, -0.7025055289268494, 1.9705686569213867, -0.23167693614959717, -0.6961926221847534, 0.14692699909210205, 0.4944533705711365, -0.5218193531036377, -3.2572097778320312, -2.0210039615631104, -0.3557399809360504, -0.5188977122306824, -2.4005043506622314, 0.26750537753105164, -1.8804445266723633, -0.6236379742622375, -0.55844646692276, -2.6408016681671143, 0.0015479708090424538, 0.8055580258369446, 1.1058212518692017, -0.07043570280075073, 0.3933253586292267, -0.5541543364524841, -0.7311894297599792, 0.21858178079128265, -0.5713354349136353, -0.43985843658447266, 0.8116983771324158, 2.2616660594940186, 0.38209810853004456, -1.0770456790924072, 0.6745290756225586, 0.04236903786659241, 0.8612199425697327, -0.8926749229431152, -0.7373232245445251, -0.9555475115776062, 0.9160919189453125, 0.1781686693429947, 0.7484280467033386, -0.7683591246604919, 0.4649398624897003, -0.018806666135787964, 0.22419756650924683, 0.3365533947944641, 1.8873114585876465, -0.07117422670125961, 0.6517272591590881, -1.3296618461608887, -0.7896947264671326, -0.6658902764320374, -1.4123178720474243, -0.3968273401260376, 2.859797954559326, -0.3954983949661255, -2.19625186920166, -0.5808613300323486, -0.39500054717063904, 0.6322311162948608, 0.29158443212509155, -2.1494600772857666, -2.177021026611328, -0.6816555857658386, 0.10764439404010773, -0.6239213347434998, -0.06345345824956894, -0.1267656683921814, -2.6299619674682617, -0.7851427793502808, -0.9035110473632812, -0.28056812286376953, -1.3585699796676636, 0.6666308045387268, -0.3124876320362091, -0.4381568729877472, 1.3427447080612183, -1.5167443752288818], [-0.2215740829706192, -2.291384220123291, -0.014899848029017448, 2.1588332653045654, 0.3043341636657715, 0.40165019035339355, 1.6074191331863403, -0.09313253313302994, -0.32902470231056213, -1.1323885917663574, 0.7500971555709839, 1.873250961303711, -0.5189642310142517, 1.0079259872436523, -1.4774086475372314, -0.47396212816238403, -1.094202995300293, -0.3564530611038208, -0.9061371088027954, -0.39716094732284546, 1.9092427492141724, 0.5500795841217041, -0.3451583981513977, 0.2184268683195114, -0.8868812322616577, -0.05535440146923065, 0.0830814465880394, -1.4601492881774902, 0.7102997899055481, -0.3596469759941101, 2.644171714782715, -0.1933898627758026, -0.05937764793634415, -0.4269731938838959, -0.7407500743865967, -0.8692163825035095, 0.4843023717403412, 1.585830569267273, -0.6627042889595032, -0.03883548080921173, 1.8254319429397583, 2.555457592010498, 0.7717396020889282, -0.31269627809524536, 0.2432233989238739, -0.4409049451351166, 0.7041814923286438, -0.14565299451351166, -0.833142876625061, 1.050820231437683, 1.3635717630386353, -0.12158165872097015, -0.6356844902038574, 0.3218906819820404, -0.005850652232766151, -1.7564021348953247, 0.7910455465316772, 0.6050418019294739, 2.003984212875366, 0.48699435591697693, 2.154531955718994, -0.8932026624679565, -1.3060970306396484, -0.5231080055236816, 0.579922616481781, -0.34777721762657166, 0.5020788908004761, 2.6399316787719727, -1.2531464099884033, -1.0743261575698853, 0.38189437985420227, -0.2708902060985565, -1.5051851272583008, 1.0322550535202026, -0.2764369547367096, -0.7827568054199219, -1.124547004699707, -1.7031357288360596, 1.5312623977661133, 0.8507460355758667, 1.5785408020019531, -0.4392731189727783, -0.08162977546453476, 0.0013911427231505513, -0.66695237159729, 1.0996357202529907, 0.04430994763970375, -0.7061748504638672, 0.10276663303375244, -2.0983386039733887, 0.902342677116394, -1.0000311136245728, 0.7574525475502014, -0.3580382466316223, -0.5347614288330078, 2.043992280960083, -0.30840516090393066, 1.0099338293075562, 0.09959959983825684, 1.4553121328353882, -0.6787991523742676, -0.9212311506271362, 1.8266353607177734, 0.07333550602197647, 0.44496044516563416, 0.10156543552875519, 1.1113444566726685, -0.09091386198997498, 0.568839430809021, -1.5484247207641602, -0.12729991972446442, 0.2156171053647995, 1.3264391422271729, -0.5385103821754456, -1.374172329902649, 0.6804116368293762, -0.05546890199184418, 1.0080506801605225, -0.4528384506702423, 2.0231435298919678, 0.6397994160652161, 1.349319338798523, 0.31333601474761963, -0.8411933183670044, -0.20301403105258942, -0.5234503149986267, 0.9863511919975281, 0.6244893670082092, 0.7460958361625671], [0.30081552267074585, -1.2598108053207397, 1.5603716373443604, 0.28452402353286743, -0.5263012647628784, 1.5914841890335083, 0.6153970956802368, -0.2670222222805023, 1.189435362815857, 0.6945103406906128, 1.6542539596557617, -1.3808984756469727, -1.8037666082382202, 1.225434422492981, 1.3874691724777222, -0.02406451851129532, -0.20915092527866364, 0.6703044176101685, 0.16157415509223938, 0.49889105558395386, 0.010089957155287266, -0.08620552718639374, 0.5723597407341003, 0.7279741168022156, 0.05216071009635925, -0.3835507333278656, -0.29095134139060974, 0.3161473274230957, 0.41991090774536133, 0.8119631409645081, -1.4237672090530396, 0.5016992688179016, 1.158761978149414, 2.6564202308654785, 0.9010065197944641, -2.166799783706665, -0.4336884319782257, -1.2685359716415405, 0.16374538838863373, -1.0166014432907104, -0.16671524941921234, -1.0314491987228394, -0.31760913133621216, -0.3818887770175934, 0.7303217053413391, -0.3042106628417969, -1.054664969444275, 0.7609819173812866, 2.3652002811431885, -0.06614505499601364, 1.9273264408111572, 0.2379932701587677, -0.5328353643417358, -0.23383693397045135, -1.5503323078155518, 0.8754609823226929, -0.5336824655532837, 0.6658615469932556, 0.4300757050514221, 0.18060733377933502, 1.2335208654403687, 0.3878355026245117, -1.0639879703521729, -0.5153894424438477, 1.4248517751693726, -0.5514203906059265, -0.2371412217617035, -0.558814287185669, -0.019053436815738678, -1.0659116506576538, -0.25576111674308777, -0.43416765332221985, -1.540678858757019, 0.688956618309021, -2.26365327835083, -0.1986108273267746, -0.018475325778126717, 1.3462177515029907, 0.25118333101272583, -1.2868541479110718, 1.4746261835098267, -1.7628775835037231, 0.14978012442588806, 1.1843976974487305, -1.050227165222168, -0.6573511958122253, 1.0084218978881836, 0.3931291401386261, -0.1439921259880066, -0.7845368981361389, -0.37567055225372314, 1.0085784196853638, 0.3505948781967163, -0.7434171438217163, -0.02570534683763981, -1.0967774391174316, -2.457653045654297, -0.5838882327079773, -0.25603801012039185, -0.516316831111908, -1.4936379194259644, 0.4509516954421997, 0.08271612972021103, 1.122089147567749, -1.1832066774368286, 0.1525506228208542, 0.6060001850128174, 0.49477848410606384, 2.0108397006988525, -0.4724142551422119, 0.6545236110687256, 0.3822471797466278, 0.9266469478607178, -0.4611927270889282, 0.40088018774986267, 1.3482385873794556, 0.22798486053943634, 0.304510235786438, -0.3184586763381958, 1.4183284044265747, 2.6202547550201416, -2.0253050327301025, 0.5419631600379944, 1.1503254175186157, -0.35830631852149963, 1.821420669555664, 0.412914901971817, 0.08430270105600357, -0.5950837135314941], [0.6155993342399597, 0.25494980812072754, 1.3635685443878174, 0.03655129298567772, 1.4623634815216064, -0.4906719923019409, 0.3757683038711548, -0.6451334953308105, -1.2850514650344849, -0.14776010811328888, -0.7647101879119873, 0.25441327691078186, 0.011377297341823578, -2.1276278495788574, 0.3330937623977661, 0.27163800597190857, 0.5710305571556091, 1.0660308599472046, 0.5194429755210876, -0.0005918609676882625, 0.05786538124084473, 0.2541661858558655, -1.0269556045532227, -0.6265609860420227, -0.15812863409519196, -1.4904537200927734, 1.8689614534378052, -0.2907761335372925, 0.31149935722351074, 1.248897910118103, 0.9568843245506287, -0.9326212406158447, -0.044451549649238586, -0.22357702255249023, -1.0313432216644287, -2.5699777603149414, -0.1513286978006363, 0.11345800757408142, 0.4445609152317047, 1.184222936630249, -0.0945109948515892, 1.4459606409072876, 1.209075927734375, 0.5233640670776367, -0.09512476623058319, -0.6498168706893921, 1.0597785711288452, -1.0785506963729858, 1.3375873565673828, 0.07145141065120697, 0.1896131932735443, -0.8194899559020996, 1.3767637014389038, 0.15102478861808777, 0.3327983021736145, -0.3199089765548706, -0.34618452191352844, -0.3502041697502136, -0.5150824785232544, -1.258033275604248, -0.9361674785614014, 1.3938547372817993, 0.01415560208261013, 1.7570933103561401, -0.48035627603530884, -0.36718660593032837, -0.8258859515190125, -0.3179074823856354, 0.9532904028892517, 1.1219806671142578, 1.336395263671875, 0.06017756834626198, -0.7137091755867004, -1.3080357313156128, 0.38860827684402466, -1.03261399269104, -0.3340366780757904, 2.258167266845703, 0.993775486946106, -0.05164123699069023, -0.9806321263313293, 1.2709040641784668, -0.31043609976768494, 1.4170589447021484, 0.5312886834144592, 1.2376128435134888, -0.8151949644088745, 0.8510752320289612, 0.7766455411911011, -0.342046320438385, 1.3643615245819092, 0.47091951966285706, -0.05178544297814369, 1.1025524139404297, -0.8697949647903442, -0.5729328989982605, 0.4538941979408264, 0.8992230296134949, -0.8051184415817261, 0.39989975094795227, -0.1117430329322815, 2.156872272491455, -1.6200205087661743, -0.0918763056397438, -0.31149736046791077, -0.6081530451774597, -0.08896990865468979, -1.2872240543365479, 0.6582598090171814, 0.2387418895959854, 0.816927433013916, -2.195844888687134, -1.2921490669250488, 0.08480250090360641, -2.2409567832946777, -0.1484658420085907, -0.6691418290138245, 1.0226455926895142, 0.28576189279556274, -0.20130226016044617, -0.9056681990623474, 0.6596227288246155, 0.589989423751831, -0.2602897584438324, -0.2102339267730713, -0.14578880369663239, 0.07273562997579575, 0.9357050061225891, -0.6363435983657837], [-2.498106002807617, 0.0761951431632042, -1.2189018726348877, -0.6868933439254761, -0.35339245200157166, 2.3511600494384766, 0.7028077840805054, -0.036596667021512985, 0.11868258565664291, 0.48424407839775085, 0.5537789463996887, 1.3304498195648193, -1.1014623641967773, -0.06493071466684341, 0.7385129332542419, 0.5949918627738953, 0.8339931964874268, -1.242260456085205, -2.664119005203247, -0.8176286816596985, -0.21255111694335938, 1.5018038749694824, 0.48287123441696167, 0.8834875226020813, 1.2280206680297852, 2.1618380546569824, -0.2563180923461914, 0.33311983942985535, -0.2688450217247009, 1.3890832662582397, 0.8497363328933716, 0.260978639125824, 0.3781109154224396, 0.02852754108607769, -0.4346466660499573, -0.46535947918891907, -0.466126412153244, -1.20121431350708, 0.2412249594926834, 1.9686733484268188, 1.0010300874710083, 0.2887240946292877, -1.9143309593200684, -0.127498060464859, 0.1465473175048828, 2.167450428009033, -0.47292906045913696, -0.06746494770050049, -0.03130136430263519, 0.0786813348531723, -0.3811401128768921, -2.43943452835083, 2.2295610904693604, -1.300768494606018, 1.0632511377334595, -0.8139014840126038, -1.7131208181381226, -0.10535317659378052, 0.5495240092277527, 0.112083800137043, 0.4116283655166626, 0.6488624811172485, -0.24568422138690948, -1.5648107528686523, 0.05400021746754646, -0.3669684827327728, -0.43487656116485596, -0.7233192920684814, -0.11388874799013138, -0.5359191298484802, -0.6227213144302368, 0.7752655744552612, -1.6539759635925293, 0.09408657997846603, -1.4390846490859985, -2.0230071544647217, 0.2554953396320343, 0.5787895917892456, 1.5396147966384888, -2.5131099224090576, -0.03599662706255913, -0.7005226612091064, 0.4455072283744812, -1.2836049795150757, -1.8912783861160278, -1.0980851650238037, 1.3866949081420898, -0.08915168046951294, -1.038895606994629, 0.4861249029636383, -0.13305117189884186, -0.25117310881614685, 2.2797234058380127, 0.7555034160614014, -1.4256457090377808, 1.4852828979492188, 0.877217173576355, 0.712842583656311, -0.38926365971565247, -0.35864830017089844, -0.8357604742050171, 1.1063315868377686, -0.14911749958992004, 1.033145546913147, 0.33268749713897705, 0.3954443335533142, 1.6239252090454102, 0.7627456188201904, 0.7451342940330505, -0.5322228074073792, -0.9188198447227478, 2.142698287963867, 0.824878454208374, 1.0588667392730713, -0.03676236793398857, -0.3823978900909424, -0.9433685541152954, -0.11355557292699814, -0.009877840057015419, -0.5407049059867859, 0.6151653528213501, -1.131674885749817, -0.9973053336143494, 0.42769965529441833, 0.17002028226852417, -1.9242554903030396, 0.10650944709777832, 0.3970058262348175, 1.8817329406738281], [1.3352186679840088, -1.7131482362747192, 0.7074577808380127, 1.5680618286132812, -1.631714105606079, -1.450264811515808, -0.6534643769264221, -0.29871097207069397, 2.581892967224121, -2.0398004055023193, -1.469909429550171, -1.7737776041030884, -0.9854274988174438, -0.22100867331027985, 0.020518261939287186, -0.4053075909614563, -0.45093315839767456, 0.20504730939865112, -1.5073347091674805, 0.9762755036354065, -1.682320475578308, -0.39507395029067993, 0.4330349266529083, -1.388218879699707, 0.7122189402580261, -0.04490364342927933, 0.20339052379131317, 2.2745866775512695, 0.6053526401519775, 0.5031445026397705, -0.5152602195739746, 0.29640763998031616, -0.3924407660961151, 0.8062655329704285, -1.3823786973953247, -0.8148273825645447, -0.5221866965293884, -1.3337122201919556, -0.2995970547199249, -0.162901371717453, -0.22433310747146606, 0.15130019187927246, 0.7643865942955017, 1.667547345161438, 0.11932045221328735, 0.7810044288635254, -1.5081279277801514, 0.0999179482460022, 0.19726094603538513, 0.4721958339214325, -1.3987029790878296, 0.1163293868303299, -0.564964771270752, 0.1818992644548416, 2.448429584503174, -0.7682173848152161, -0.7141743302345276, 1.238878607749939, 0.059059444814920425, -0.4040374755859375, 0.8413681983947754, 0.21207858622074127, 0.36595186591148376, -0.3189225196838379, 1.1949907541275024, -0.28967368602752686, 2.6919496059417725, 0.31927621364593506, 0.41353121399879456, -0.48172473907470703, -0.8405901789665222, 0.16815882921218872, -0.47136470675468445, -2.0437307357788086, -0.17345327138900757, -1.0654890537261963, 0.08482485264539719, -1.6933379173278809, -1.274430274963379, 1.2615996599197388, 0.9946087598800659, -2.598895788192749, -0.32239478826522827, -0.45261892676353455, 0.42738544940948486, -0.20480948686599731, 0.05838240310549736, 0.44718509912490845, -1.5251307487487793, -1.5381585359573364, 0.4268399775028229, 1.2981421947479248, 0.6224011778831482, 0.4603852927684784, 0.26190948486328125, -0.4539313316345215, 1.5579358339309692, 0.7283987998962402, -1.716338872909546, 0.4489978551864624, -0.9400380253791809, -0.8194610476493835, 0.051368046551942825, -0.9018516540527344, -1.2717113494873047, 0.7835559844970703, -0.7948370575904846, 1.0677945613861084, -1.0677118301391602, -1.0342384576797485, -0.6252744197845459, 0.05022042244672775, 0.6868419647216797, -0.25322604179382324, -1.1239475011825562, -2.697223663330078, -1.3518446683883667, -0.5352083444595337, -0.3090011477470398, 1.75762140750885, -0.6099998354911804, 0.09000594913959503, 1.334834098815918, 1.5963709354400635, -0.03253833204507828, 0.2868606150150299, -0.7179191708564758, -0.37231871485710144, 0.39236822724342346], [0.03768207132816315, -0.39253443479537964, -0.6310757994651794, -0.0636061429977417, 1.0029414892196655, -0.8020164370536804, -1.2073324918746948, 0.5610112547874451, -0.9707419276237488, -0.6949774026870728, 1.3011233806610107, 1.4347584247589111, -0.27310216426849365, -0.1844293475151062, -0.06505456566810608, 0.09697752445936203, -0.38906756043434143, 1.7873448133468628, -1.6323641538619995, 0.5061602592468262, 0.8974413275718689, -0.8982111215591431, 0.5325998067855835, -0.6923159956932068, -1.5231051445007324, -0.6807482242584229, 0.6481927633285522, 1.7962886095046997, -0.4286993443965912, -1.1459177732467651, 1.6092513799667358, 0.8938519954681396, 1.7869938611984253, 0.2851824164390564, -1.60377836227417, -0.8598892688751221, -0.09673035144805908, 1.4293084144592285, -1.2645515203475952, 1.105086326599121, 1.663823127746582, 0.5490983128547668, -0.06871307641267776, -0.8574890494346619, -0.7974580526351929, -1.2586511373519897, -1.0901981592178345, -0.9511503577232361, 1.144410252571106, 0.1905759871006012, 1.0045615434646606, -0.8992645740509033, -1.3919023275375366, -0.9466982483863831, 0.6040137410163879, -0.03242800384759903, -1.5050005912780762, -0.6240912079811096, 0.7053133845329285, 0.39825448393821716, -0.45583587884902954, -2.1057305335998535, 0.32475724816322327, -0.1646391898393631, 0.6053741574287415, 0.15919367969036102, 0.3186391294002533, 1.189996600151062, -0.08984333276748657, -1.4311909675598145, -0.9536358714103699, 0.3957851529121399, 1.5125341415405273, 0.3648209869861603, 0.2258162498474121, 0.9886994957923889, -0.5233590006828308, 0.8742718696594238, -0.39772137999534607, 0.43877527117729187, 0.4610932171344757, -0.09411637485027313, 0.24275989830493927, 0.77840256690979, 0.40287351608276367, -1.1738054752349854, -0.4772854447364807, 0.7805043458938599, -0.3338640630245209, 1.113184928894043, -1.1539175510406494, -0.07675579190254211, 0.23650503158569336, 1.0186935663223267, -0.5867522358894348, -0.3493310511112213, -0.5709567666053772, -1.1957110166549683, -0.22826877236366272, -1.2436366081237793, -0.004096328746527433, 0.8445338010787964, -0.265601247549057, 0.29742512106895447, 0.8890386819839478, 1.2437551021575928, 0.9343546628952026, -1.8717410564422607, 1.608493685722351, 0.13783252239227295, 0.7534263134002686, 0.09650612622499466, 0.27248623967170715, -0.6512459516525269, -1.0732603073120117, -1.5802278518676758, -0.3393225371837616, -0.7572398781776428, 1.0712231397628784, -1.1111294031143188, -0.987953782081604, -0.18737806379795074, -0.4785659611225128, -0.9403003454208374, -0.5630300045013428, 0.9756774306297302, -1.42800772190094, -0.027539890259504318, -0.11923246085643768], [-0.25351351499557495, 1.5878863334655762, -0.8726654648780823, 1.496626377105713, 0.589844286441803, -0.320069819688797, -0.14312295615673065, -0.29476916790008545, -0.020087797194719315, -0.3151739239692688, 0.5607486367225647, -0.9780736565589905, -1.2243438959121704, -0.07581640779972076, -1.2621517181396484, 0.4985896348953247, 1.1899479627609253, 0.5595793128013611, 0.46105414628982544, 0.1866377890110016, -2.1967389583587646, -0.8964551091194153, -0.12454428523778915, -0.277482807636261, 2.037553548812866, 1.158036708831787, 1.1942578554153442, -0.9436241984367371, 0.2904423773288727, 0.3294326364994049, -0.05058705061674118, 2.584709644317627, -0.2740533649921417, -0.8741220235824585, -1.62674880027771, -1.17415452003479, 0.6587281227111816, -0.2420661449432373, -0.699155867099762, 0.30998730659484863, 0.2927931547164917, 0.14260733127593994, -1.470026969909668, 0.9050076007843018, -0.4947986304759979, 1.0338170528411865, -0.8659661412239075, -0.6134046316146851, -1.3653149604797363, -0.7109978795051575, -0.6097217202186584, -0.8160322904586792, -0.5936470627784729, 0.8025925159454346, -0.1757868528366089, 0.8676884174346924, -1.3017997741699219, 1.3618056774139404, -1.4823282957077026, 1.1424764394760132, -0.47054624557495117, 0.17313414812088013, 0.44858890771865845, 0.03577357158064842, 1.5158793926239014, -0.02795449271798134, 0.06216355040669441, -0.6000719666481018, 0.6076227426528931, 0.19300507009029388, -0.10811185836791992, -1.6971979141235352, 0.7188182473182678, -1.2263864278793335, -0.012298858724534512, 0.9591262936592102, 0.08938118815422058, -1.0354784727096558, 0.27095043659210205, -0.11033352464437485, 1.153558373451233, -1.648118019104004, -2.2783620357513428, 1.0056337118148804, -0.41453540325164795, -1.326928734779358, 0.5440338253974915, 0.011167451739311218, 1.1760249137878418, 0.21040841937065125, 0.5864867568016052, 0.36226582527160645, 1.5286471843719482, -1.0101182460784912, -2.4311490058898926, 1.4214717149734497, 0.007021255791187286, -0.5035288333892822, -0.6735772490501404, -0.7052080631256104, 0.058859821408987045, -1.9919805526733398, 2.34660267829895, -0.18269450962543488, 0.0884656310081482, 0.7846127152442932, -0.6858698129653931, -1.7605066299438477, 1.3703374862670898, 1.3511614799499512, 0.8362993001937866, -0.4147173762321472, -0.5185847282409668, -0.6909805536270142, -0.6359620690345764, 0.34774112701416016, -0.08675547689199448, -1.1745721101760864, 0.8288434147834778, -0.4837629199028015, 0.1196959838271141, -1.7381280660629272, -0.5714460611343384, -1.4848495721817017, -0.48030659556388855, 1.1481484174728394, 0.29423651099205017, -1.2513477802276611, 0.37750327587127686], [1.0190175771713257, -1.6545082330703735, 0.8541002869606018, -1.2401946783065796, -0.1833893358707428, -1.0117311477661133, -0.8752731084823608, -1.1011691093444824, -0.1553328037261963, -1.229879379272461, 0.07770204544067383, 0.44959694147109985, 1.1992653608322144, 0.8392373323440552, 0.5024117231369019, 0.8142377734184265, 0.7987872958183289, -1.5776902437210083, -0.693533182144165, -1.9370137453079224, -1.1810046434402466, 0.25063085556030273, -0.11736934632062912, -1.2054429054260254, 0.26962602138519287, -1.6689308881759644, 0.1553889513015747, 0.06371089071035385, 0.1899091899394989, 1.2284849882125854, -0.8419368863105774, -0.192155122756958, -1.435746192932129, 0.36725783348083496, -0.21823756396770477, 0.5808461308479309, 0.2920442819595337, 0.735174834728241, -0.04863312840461731, -0.5653754472732544, 0.8542463779449463, 0.8423847556114197, -0.7016790509223938, 0.2878859043121338, 0.4151638448238373, 0.18568122386932373, 0.671554446220398, 1.0070536136627197, -0.2487894892692566, -0.7625333666801453, -1.313966989517212, -1.0449515581130981, 0.6022081971168518, 0.09329186379909515, -0.06366211175918579, -0.838560163974762, 0.39118048548698425, -0.1291622668504715, 0.8442538380622864, 0.125020831823349, 0.708021342754364, -0.1955559402704239, -0.5094563961029053, -0.8403534293174744, -0.42946121096611023, -0.3535976707935333, 0.5868088603019714, -1.2355586290359497, -0.3891826570034027, 0.8193081617355347, -1.4437834024429321, 1.6008270978927612, -1.3835906982421875, -1.6007887125015259, -1.4024617671966553, -0.3297071158885956, -0.0969398245215416, -0.3936370611190796, 1.0193068981170654, 0.2322380095720291, -1.0768883228302002, 0.6433168053627014, -0.023692846298217773, -0.5282354950904846, -0.43773090839385986, 0.3916209638118744, 0.9771611094474792, -0.7851309180259705, 0.15292546153068542, 0.05579586699604988, -0.54960697889328, 0.9255969524383545, 0.3210451304912567, 1.4107189178466797, -0.9985789656639099, 1.1700135469436646, -0.7968025207519531, 0.3121398985385895, 1.4029232263565063, 0.4556070864200592, 0.9303770065307617, 0.3632642924785614, -0.6416589021682739, -0.5438003540039062, -0.40415525436401367, 0.029889244586229324, 1.426870584487915, -0.5476645231246948, -0.6667787432670593, -2.0268912315368652, -0.556087851524353, -0.07965045422315598, 0.753561794757843, -0.8490956425666809, 0.35296759009361267, -1.2122135162353516, 1.0898288488388062, 0.7475610971450806, 0.0001312944368692115, -0.19740305840969086, 1.1929564476013184, -1.0126298666000366, -0.9854928851127625, -0.44059082865715027, -0.3792782127857208, -0.9834353923797607, 0.30945080518722534, 0.2167329043149948, -0.0004136773641221225], [-2.0340499877929688, 0.3266969621181488, -0.4319557845592499, 0.05777674913406372, -0.9646178483963013, 0.5928516983985901, 0.7154353857040405, -1.3989726305007935, -1.2999309301376343, 1.2600079774856567, 1.1286137104034424, -0.0754101574420929, 0.9484631419181824, 0.5462629795074463, 2.4963417053222656, -1.0919617414474487, 1.1972377300262451, -0.6087223887443542, -0.0902857854962349, 0.8840821981430054, 1.2712682485580444, 0.31945115327835083, -0.4157526195049286, 0.050310492515563965, -2.2029969692230225, 0.012549817562103271, 0.4576627314090729, -0.8932348489761353, -0.4390873312950134, 0.7432090640068054, -1.126023530960083, -0.7977745532989502, 2.069955825805664, 0.30539223551750183, -0.873297393321991, -0.916972279548645, -1.0319726467132568, 2.2173843383789062, 0.751288890838623, -0.7855452299118042, 0.8950697779655457, -0.36011800169944763, 0.2464114874601364, -0.41034117341041565, 1.5683419704437256, 0.6573576927185059, 1.5011144876480103, 0.3262682855129242, -1.9345422983169556, -0.7990819215774536, 0.03242265060544014, -0.9357612133026123, -0.14029712975025177, -0.6668786406517029, 0.13811343908309937, 1.365787386894226, -0.6958058476448059, -0.3506568670272827, -1.5899299383163452, 0.3912573456764221, -0.2600160539150238, 0.5352662801742554, 0.46752163767814636, 0.10079335421323776, -2.1055498123168945, -0.23399806022644043, 0.1338495910167694, 0.9981119632720947, -0.3107985854148865, 0.28725337982177734, -0.46606457233428955, 1.8720195293426514, -0.2696506381034851, -0.3711318075656891, 2.0164856910705566, 0.03194402530789375, 1.005920171737671, -1.2502532005310059, -1.0263203382492065, 1.3244011402130127, -0.2302858829498291, -1.0479544401168823, -0.9256885051727295, -1.5679837465286255, -0.6862030625343323, -1.5976521968841553, -0.9268599152565002, 1.1875909566879272, -0.8178274035453796, -2.0113131999969482, 1.3901875019073486, 0.11296127736568451, -0.8935691714286804, -0.5734961628913879, 0.3175436556339264, -0.6823747158050537, -0.28267794847488403, -0.1457788348197937, -0.21426646411418915, -1.4797186851501465, -1.061972975730896, -1.320080041885376, -0.8291346430778503, 0.7733574509620667, -1.5911333560943604, 0.08442071825265884, -2.4591522216796875, 1.3188050985336304, -1.3093515634536743, -0.5029059052467346, -1.0700522661209106, -0.0019773123785853386, -0.2450762391090393, 0.7007783055305481, -2.048074722290039, 0.501560628414154, 0.6343593001365662, -0.6254923343658447, -1.0794093608856201, 0.3337164521217346, 0.3409712612628937, 2.0757250785827637, 1.668047308921814, -0.28281208872795105, -0.4451868534088135, -0.4818456172943115, -0.5916516184806824, -0.2774384319782257, 0.9733898639678955], [0.41461819410324097, 0.6042847037315369, -1.3908544778823853, -0.49361878633499146, -0.46679601073265076, -2.1484169960021973, -1.7164497375488281, -1.7505255937576294, 1.5369176864624023, -0.1576647311449051, 0.8942839503288269, 1.1639328002929688, -1.44159734249115, -1.441384196281433, -1.575858473777771, -0.4998757243156433, -1.237937092781067, 1.1711634397506714, -0.8443843722343445, -0.24352160096168518, 1.0938003063201904, -1.493569016456604, -0.6062449216842651, 1.7501879930496216, -0.8461403250694275, -0.3637806177139282, -0.8708568811416626, -0.8412688374519348, 0.19756536185741425, -0.6329155564308167, 1.9680436849594116, 0.7271549701690674, 0.344919353723526, -0.23165427148342133, -0.8491277098655701, -1.3686715364456177, -0.7221812009811401, 0.05769771710038185, 0.48044532537460327, -1.6363825798034668, 1.4936225414276123, 0.5865007042884827, 0.23789267241954803, 0.21966011822223663, 0.14977121353149414, 0.8572681546211243, -0.4640360474586487, 0.12644599378108978, -0.9141191840171814, 0.5467196106910706, -1.2061727046966553, 0.5003371238708496, -1.1011039018630981, -0.6324503421783447, -0.5263996720314026, -1.1907323598861694, 0.719822108745575, 0.08491741865873337, 0.501849353313446, -0.6382172107696533, -0.04292403534054756, 0.5067397356033325, -0.7946746945381165, 0.7060202360153198, -1.2478362321853638, -1.091018795967102, 1.0242836475372314, -0.3602636754512787, -0.2218402624130249, 0.4313434660434723, -2.119492769241333, 0.5897956490516663, -0.3486250042915344, -2.420156478881836, 0.2650134563446045, -1.243066430091858, 1.185856819152832, 1.4915857315063477, 2.325488567352295, 1.3931065797805786, 0.13328956067562103, -0.8417643308639526, 0.9259442687034607, -0.31421247124671936, -0.35843315720558167, 1.0446455478668213, -0.048305001109838486, -0.6819821000099182, 1.667517900466919, 0.1489916741847992, 0.7389736771583557, -1.1723769903182983, 2.720571994781494, 0.0008489196188747883, -1.4466443061828613, 0.45311877131462097, -1.267695665359497, -0.147018700838089, 1.6427983045578003, -0.2851874828338623, 0.9004374146461487, -0.1982157677412033, 1.839560866355896, -0.4797745943069458, -1.2110508680343628, 2.0603830814361572, 0.11987233906984329, -0.42957940697669983, -1.1885844469070435, 0.9237494468688965, 0.5717038512229919, 0.26248788833618164, -1.7830909490585327, 1.6650699377059937, 0.3464718461036682, 0.030901717022061348, -0.5938687920570374, -0.41229578852653503, 0.36997857689857483, 0.46315962076187134, -0.046356990933418274, 1.6981208324432373, 0.8125876784324646, 0.30863603949546814, -0.5161914229393005, -0.3008226752281189, 0.11158480495214462, -0.811604380607605, 0.2764895260334015]], "dtype": "float32"}, {"__nd__": [1.1598284244537354, 0.17679278552532196, 0.31292611360549927, 0.3524201512336731, -0.22352638840675354, -0.2799234688282013, 1.209466576576233, -1.4808660745620728, -0.1130061000585556, 0.45642057061195374, -0.013848927803337574, -0.839611828327179, 0.46684348583221436, -0.6622520089149475, 0.5322475433349609, -0.815528929233551, 1.6745238304138184, -0.39517658948898315, 1.3643585443496704, -0.4196910560131073, -1.0760585069656372, -0.21690721809864044, 0.41784733533859253, 0.8024905920028687, -1.430326223373413, -0.40525421500205994, -1.0027304887771606, -0.8114686012268066, 0.9940358400344849, 0.1344035118818283, -2.4866702556610107, 0.2997373342514038, 1.9723684787750244, 1.2004499435424805, 0.6158308386802673, 2.6312758922576904, -0.25943535566329956, -1.4654567241668701, 0.05081517994403839, 1.3248258829116821, 0.8700470328330994, -0.5755178332328796, 0.5638406276702881, 0.3472009003162384, 2.2580015659332275, -0.09032946825027466, -1.6625850200653076, -1.0239862203598022, -0.7983274459838867, 0.4551939368247986, -0.09151347726583481, 0.7767047882080078, 0.8520383834838867, -0.9614652395248413, 0.6599470376968384, -0.062199316918849945, 1.0819289684295654, -0.2680681049823761, 0.6209293603897095, -0.7357406616210938, 0.5896676778793335, 0.5103844404220581, -0.13302186131477356, -0.03381824120879173, -1.0972334146499634, 0.9737605452537537, -0.4008807837963104, -0.6485469341278076, 0.8707271218299866, 1.4424124956130981, -1.1721776723861694, -0.839892566204071, -2.2292356491088867, 1.5560288429260254, -0.17071682214736938, -1.608007550239563, -1.6594161987304688, 2.0499768257141113, 1.1539732217788696, 2.050523281097412, -0.11459033191204071, 0.7172259092330933, 1.6403861045837402, 0.47194141149520874, 0.672543466091156, -0.9662564396858215, 0.14054208993911743, -0.4642886221408844, 0.987184464931488, -1.089111089706421, 1.7045220136642456, -0.3753509223461151, 2.045252799987793, -1.005371332168579, -0.3626573085784912, 0.030152706429362297, -1.4610576629638672, -0.06566206365823746, -0.1579037308692932, 0.07535728067159653, 0.07883204519748688, 0.24950993061065674, 0.05641430988907814, -0.5115607976913452, -0.6533652544021606, -1.2925639152526855, -0.7427563667297363, -0.3632873594760895, 0.24313409626483917, -0.4396767318248749, 0.0206642746925354, -1.4598897695541382, -1.204915165901184, 0.3601651191711426, -1.542860984802246, 1.4785536527633667, 1.5661752223968506, 0.4417843222618103, -0.758729100227356, -1.0120911598205566, -1.75446355342865, -0.0010660849511623383, -0.5555665493011475, -0.918119490146637, 0.24623461067676544, -0.5126568078994751, 1.8826736211776733, 1.8297520875930786, 0.2522476613521576], "dtype": "float32"}, {"__nd__": [1.0889875888824463, -0.2633489668369293, 0.6414638757705688, 0.28095680475234985, 1.0617575645446777, 0.4808664917945862, -1.5842833518981934, 1.7551552057266235, -1.0566127300262451, 0.9656792283058167, -0.07054591178894043, -0.679965078830719, 1.1690781116485596, -0.17220811545848846, -2.2470734119415283, -1.8529895544052124, 0.029711151495575905, -0.07670923322439194, -0.7956622838973999, 0.13639310002326965, 1.1463608741760254, -0.882219672203064, 0.056600362062454224, -2.388338565826416, -0.24115389585494995, 0.4710918366909027, 1.7283655405044556, 0.7213677167892456, -1.133540391921997, -1.6039361953735352, -0.26678231358528137, 1.360482096672058, 1.3575340509414673], "dtype": "float32"}, -1.5, 2.0], "e": {"__nd__": [-33.238922119140625, 23.537208557128906, 4.92559289932251, -14.537487030029297, 14.704601287841797, 19.75908660888672, 21.02730941772461, -9.95429801940918, 1.3598359823226929, -31.607622146606445, 11.887354850769043, 14.43488883972168, 13.133834838867188, 6.2320356369018555, 29.9403018951416, 11.925326347351074, -13.959552764892578, -6.405088424682617, -1.1249985694885254, -12.80234432220459, -31.727188110351562, -10.384467124938965, 19.973785400390625, 0.49689874053001404, -11.539613723754883, -24.533967971801758, 10.988354682922363, 24.99524688720703, 35.289615631103516, 12.518199920654297, -7.810283184051514, -11.775659561157227, -10.484599113464355], "dtype": "float32"}}, {"a": [{"__nd__": [[0.0917675569653511, -0.28705886006355286, 0.6371315717697144, 1.376326084136963, -0.11982352286577225, -0.030755111947655678, 1.4735069274902344, 1.096029281616211, -1.0036097764968872, -0.9642956852912903, 0.06400544941425323, -1.7898311614990234, 1.629799246788025, -0.5860339403152466, 0.22040866315364838, 0.026574697345495224, -2.052520990371704], [0.46406784653663635, -1.1861382722854614, -0.0543750636279583, 1.6435648202896118, -0.5085362792015076, 0.8500588536262512, 0.7190895676612854, 0.2105104923248291, -0.19635868072509766, 1.4152752161026, 0.27559539675712585, 2.0564932823181152, -0.7235327959060669, -0.984495997428894, 1.6690068244934082, -0.7943681478500366, 0.16024208068847656], [-0.3216480314731598, 0.09557730704545975, 0.08433111011981964, 0.7973394393920898, 0.23549756407737732, 1.9993016719818115, 0.9239803552627563, 0.1211942732334137, -1.2630189657211304, 0.7503231763839722, -0.9307103753089905, 0.012679592706263065, -0.1191011369228363, 0.2274235188961029, 1.1581075191497803, 1.0444176197052002, -0.328460693359375], [1.406799077987671, 0.03623852878808975, 1.7816568613052368, 0.9958865642547607, 0.9321792125701904, -0.30438828468322754, -1.2833807468414307, 0.22617800533771515, -0.30185291171073914, 1.34523344039917, -0.08736363798379898, 1.1706308126449585, 0.0915721207857132, -0.9502831697463989, -0.8989320397377014, 1.1780279874801636, -1.108040452003479], [-0.6321602463722229, -0.47006991505622864, 0.6080735325813293, 0.6249375939369202, 0.4205990135669708, 0.2363165020942688, -0.015907742083072662, 1.5994654893875122, -1.5943385362625122, 0.13249756395816803, 2.1760246753692627, 0.5303801894187927, -0.2661212980747223, 2.0067834854125977, 1.3792881965637207, -1.3579825162887573, -0.2152581512928009], [0.05874739587306976, 0.1988397091627121, 0.04700605198740959, 0.8952741026878357, -1.7322245836257935, -0.834730863571167, 0.5708147287368774, 0.3740236163139343, 0.018025102093815804, 0.6911422610282898, 0.6999170184135437, -0.3351282477378845, 0.25445234775543213, -1.5952714681625366, 0.8188004493713379, 0.3886321783065796, 1.022043228149414], [0.10880530625581741, -0.239307701587677, -0.4751889407634735, 1.2101197242736816, -1.9175611734390259, -1.5596307516098022, 0.20306435227394104, 0.39779195189476013, 0.8588029742240906, 0.02549232542514801, 1.1561745405197144, 0.24836856126785278, 0.5136454105377197, 1.1067758798599243, -1.3893506526947021, -1.3886536359786987, -0.9515110850334167], [0.7800025939941406, 1.3701492547988892, 0.42353302240371704, 0.8907973766326904, -0.012197257950901985, -1.6457170248031616, 0.01948392018675804, 0.2969696819782257, 1.5589598417282104, -0.39275506138801575, -2.876002311706543, 0.547210156917572, 0.3303896188735962, -1.2632935047149658, -2.276041269302368, 1.5566514730453491, -0.5320422053337097], [-1.5220316648483276, -0.3621602952480316, 0.1336900144815445, -0.87035071849823, -0.5250172019004822, 0.38918623328208923, 2.801136016845703, 1.4451919794082642, -0.4219404458999634, 0.021381882950663567, -0.44100797176361084, -0.13818275928497314, 0.5722829699516296, 0.6100531220436096, 0.12836049497127533, 1.497243881225586, 1.1438069343566895], [0.3604806363582611, 1.242444634437561, -1.942762017250061, -1.715410828590393, 0.3405478298664093, -0.0039824554696679115, -0.5130065083503723, 0.23005707561969757, 1.2174463272094727, 1.025228500366211, 1.3698490858078003, -0.7931607961654663, 0.7215642333030701, 0.7628985643386841, 1.0765535831451416, -0.3642221689224243, -1.8461312055587769], [-2.3931758403778076, 1.4203826189041138, 1.8020230531692505, -0.08410485088825226, -0.6036084294319153, -0.7133294939994812, -0.4767768979072571, -0.3215102255344391, -0.26231133937835693, 0.49376121163368225, -0.18994863331317902, -1.730285882949829, 0.020708726719021797, -0.16027306020259857, -0.46038252115249634, 0.9939587712287903, -0.2604573369026184], [0.08269218355417252, 0.09952191263437271, 1.203808307647705, 0.7343132495880127, 1.2242759466171265, -0.2004152536392212, -0.030758410692214966, -1.2664852142333984, 0.07025036960840225, 0.6655998826026917, -1.4802324771881104, -0.2629578411579132, -0.349365770816803, 2.1586148738861084, 0.9449743032455444, -1.4308063983917236, -1.835941195487976], [0.6665031909942627, -0.505591094493866, -0.13858841359615326, -0.349804162979126, 0.6871841549873352, 0.48183998465538025, 0.7323727607727051, -0.023494558408856392, -1.0889843702316284, 1.6707074642181396, -0.46819135546684265, 0.8246433734893799, -0.3038282096385956, 0.0353158600628376, -0.9498932957649231, 1.1363351345062256, -2.589442729949951], [0.21860791742801666, 1.5786685943603516, 2.0068018436431885, 0.8520369529724121, -1.4204561710357666, -0.9436350464820862, -1.321993112564087, 0.1617010086774826, -0.10382585972547531, -0.3318837881088257, 0.3018551766872406, -1.4474714994430542, -1.481554388999939, 1.7086482048034668, -0.4604436159133911, 0.9132745265960693, 0.21470431983470917], [0.21244515478610992, -1.3601269721984863, 0.9304605722427368, -0.7483265399932861, -0.3407016694545746, -0.04095330461859703, -0.92659991979599, 1.9983989000320435, -1.8122435808181763, 1.0588452816009521, -0.8303731083869934, 0.9382503628730774, -1.8087666034698486, 1.8381261825561523, -0.3270357549190521, -0.16760915517807007, 0.4036155343055725], [1.254042387008667, -0.19127844274044037, 0.8684274554252625, 1.5333225727081299, -0.7410715818405151, 0.30980125069618225, -0.5246888399124146, 0.5063098073005676, -0.007619577459990978, -0.6843374371528625, 1.9286247491836548, 0.8852608799934387, -0.6651298403739929, -0.6319032907485962, 0.8254634141921997, 0.20157405734062195, 1.4523584842681885], [-0.23899777233600616, 0.6625198721885681, -0.4255043864250183, 0.7305601239204407, -1.016189455986023, -0.9748254418373108, 0.3121795654296875, -0.058010000735521317, 1.6149535179138184, 0.27051684260368347, 1.7918275594711304, 0.9349246621131897, -0.5347418785095215, 0.9944100379943848, -0.6193429231643677, 0.4099481999874115, -1.0689988136291504], [0.7025811672210693, -0.527737021446228, -1.3773503303527832, 0.7981617450714111, 0.34268683195114136, 0.6505631804466248, 1.8688557147979736, 0.9587652683258057, 0.496574729681015, -0.9160529971122742, 1.2367240190505981, -2.1645233631134033, 1.6214499473571777, 0.46003443002700806, -1.3806982040405273, 0.30270108580589294, 0.8891381621360779], [-0.14267320930957794, 0.7729280591011047, -0.6150851249694824, 0.6021955609321594, -1.4068562984466553, 0.21813571453094482, -0.45989200472831726, -0.3849129378795624, -0.47279593348503113, 0.2849791347980499, -0.4533974230289459, -1.5269362926483154, -0.6399059295654297, 0.559322714805603, -1.249311089515686, 0.8520647883415222, -1.1870554685592651], [-0.19763176143169403, -1.225345253944397, 0.3714505732059479, -0.21533022820949554, 0.014962016604840755, -0.9993559122085571, 0.1858738511800766, -0.11092016845941544, -2.5278735160827637, 0.9273101091384888, -0.9863350987434387, -0.19429229199886322, -0.326282262802124, -0.6896814703941345, 1.8629462718963623, 0.6704787015914917, -1.390075922012329], [-0.19291289150714874, -0.22585119307041168, -0.7192477583885193, 0.2829837203025818, 1.4900200366973877, -0.6987134218215942, -1.2433409690856934, 1.2159305810928345, 0.744662880897522, -2.42622447013855, 1.0734232664108276, -0.270011305809021, 1.9428633451461792, 1.42577064037323, 0.42853349447250366, 0.32886430621147156, 0.505591094493866], [-2.622469425201416, -0.1353009194135666, 0.2944222092628479, -2.0630998611450195, -0.2681889832019806, -1.1982765197753906, -0.6764286160469055, 0.17499969899654388, 0.3245149850845337, 0.09532807022333145, -0.3080040216445923, -0.9150206446647644, -0.4952285885810852, -1.0024666786193848, -0.8837681412696838, -0.7229110598564148, -1.6739730834960938], [0.42089006304740906, 0.14834459125995636, 0.008238257840275764, -0.7554742097854614, 1.4395310878753662, -0.329969584941864, -0.47369900345802307, 0.12248757481575012, -0.29994213581085205, -2.556243896484375, 0.03671133518218994, 1.207287311553955, 1.1065322160720825, -1.3181928396224976, -1.0267771482467651, -0.7798664569854736, -0.3606202006340027], [1.4233686923980713, 0.6032170057296753, 0.4287470579147339, 0.6169702410697937, -0.5709840655326843, -0.21957334876060486, -1.363119125366211, -0.09816200286149979, -0.8884190917015076, -0.9113417267799377, -0.3079438805580139, 0.8212286233901978, 0.9319918155670166, -0.33086174726486206, 0.08849188685417175, -0.7700921297073364, -0.10173285007476807], [1.528687834739685, 0.31766635179519653, -0.614754319190979, 0.26819220185279846, 0.20779164135456085, -0.5598660707473755, 1.0346801280975342, -0.30668604373931885, 0.7200424075126648, -1.9276049137115479, 0.394057035446167, 0.11176140606403351, 0.49627208709716797, -0.6253005862236023, -0.5521900653839111, 1.1721118688583374, -0.24648071825504303], [1.797191858291626, -0.17818917334079742, 0.1363905817270279, 0.1988985687494278, 1.5072486400604248, 0.09720343351364136, 0.21223010122776031, -1.1633936166763306, 1.8867192268371582, -0.9057245254516602, -0.1036367118358612, 1.8518126010894775, -1.4498132467269897, -0.2637620270252228, -1.3132563829421997, 0.8572105169296265, -2.2216904163360596], [0.3108210265636444, -1.4185575246810913, -0.061958085745573044, -1.0147924423217773, -0.6468448042869568, -0.7632262706756592, 0.556610643863678, -0.9598044157028198, 0.12681813538074493, -1.4586124420166016, 0.34863027930259705, -0.451042503118515, 0.3192610740661621, 0.3395121991634369, -0.10846533626317978, 1.4289588928222656, 0.9636171460151672], [0.546281099319458, 0.8770554065704346, 1.5942262411117554, -1.05406653881073, 1.1297601461410522, -0.8459420204162598, 0.3367347717285156, 1.0682717561721802, -0.5013669729232788, 0.3968663215637207, -0.8528162240982056, -1.044523000717163, -0.05226830020546913, 1.7554371356964111, -0.41041117906570435, 1.0102558135986328, 0.3869255483150482], [-0.6095162034034729, 0.7721814513206482, 1.2076741456985474, 0.5798264741897583, -0.5167349576950073, 0.4056767523288727, -1.3210209608078003, -0.3812237083911896, -0.8051659464836121, -0.6017087697982788, 0.6928471922874451, -1.1398632526397705, 1.3931236267089844, 0.2303982973098755, 0.2650357484817505, 0.3357265889644623, 0.48295289278030396], [0.37183013558387756, 1.38125479221344, 1.049716591835022, -1.1385514736175537, -0.7723690867424011, -0.34950926899909973, -1.9239877462387085, 0.15846198797225952, -0.8888810873031616, -0.022357238456606865, 1.2005248069763184, -0.6488789916038513, -2.7081596851348877, 0.11136089265346527, 0.918684720993042, -1.5645875930786133, 0.1629980206489563], [1.1686252355575562, -0.5335233211517334, 0.06042240187525749, 0.8267539143562317, -0.7058596611022949, -0.9157143235206604, -0.48910847306251526, -0.7295633554458618, 0.6055567860603333, -0.6390388607978821, 1.2123517990112305, 0.12004423886537552, -0.5008971691131592, 0.9645198583602905, -1.2030348777770996, 1.4792695045471191, 0.5633844137191772], [-0.7690668702125549, -0.5177061557769775, -1.8916475772857666, 0.4956570267677307, -1.618062973022461, 0.13431021571159363, -0.04188959673047066, -1.610310435295105, -0.3529641628265381, -0.2624404728412628, 0.8262816071510315, -1.279966115951538, -1.5048785209655762, -0.14654260873794556, 0.7232035994529724, 0.29008305072784424, -1.159285068511963], [1.0690484046936035, 1.2130190134048462, -0.3597792387008667, 0.6971349716186523, -0.44061145186424255, 0.7854543924331665, -1.3907936811447144, -1.9678281545639038, 0.05973999947309494, -0.5636612176895142, 1.8391358852386475, -0.9548938274383545, 0.5775639414787292, 1.3633147478103638, 0.8957725763320923, -0.17526498436927795, -0.1589106023311615], [0.5245140194892883, -0.05310098081827164, -0.26143679022789, 0.046262167394161224, -0.2954464256763458, 0.6535338163375854, -1.7841469049453735, 2.9992120265960693, -0.11688212305307388, -0.6087135076522827, -0.5193597674369812, -0.05568072199821472, -0.10972092300653458, 0.494625449180603, 0.5844703316688538, -0.9402391910552979, -0.49103018641471863], [0.5359904766082764, 0.42177078127861023, -1.5056958198547363, 0.9694984555244446, -1.5734405517578125, 1.2166218757629395, 0.9971746206283569, 0.5397850275039673, -0.19366733729839325, 0.6544777154922485, 0.11920934915542603, -0.0685550719499588, -0.873127281665802, 1.5711989402770996, 1.2134917974472046, 0.649402916431427, 1.7809678316116333], [2.2847673892974854, -0.5024875998497009, 2.093628406524658, 0.7313539385795593, 1.8016666173934937, 0.1739160567522049, -0.17855708301067352, -0.006635648664087057, -0.5950374603271484, -0.5014921426773071, 1.6491907835006714, 0.516622006893158, 1.2010730504989624, 0.4019622802734375, -1.0268497467041016, 0.08596255630254745, -1.368864893913269], [-0.7971178293228149, 0.797961413860321, -0.0688493400812149, -0.33697277307510376, -0.3374689221382141, -0.5087811946868896, -0.2686326801776886, -1.30442214012146, 0.1404029130935669, -0.8166803121566772, 1.2911791801452637, -0.2535606622695923, -0.9539769887924194, -0.42626047134399414, 0.3641548752784729, 0.7291229963302612, -1.417082667350769], [0.5875853896141052, 0.757169246673584, 1.22710120677948, -0.22204481065273285, 0.914150059223175, 0.01802760362625122, 0.13864120841026306, 0.7323581576347351, -1.091577410697937, -1.7374753952026367, -1.1917753219604492, 0.7710353136062622, -1.6975724697113037, 0.11690951138734818, -0.3334752321243286, -0.7632051110267639, 0.8118635416030884], [1.2568496465682983, -0.8015341758728027, 1.4225367307662964, 0.8237994313240051, 1.2389838695526123, -2.4063124656677246, 0.19170135259628296, 0.9513923525810242, -1.254744291305542, -2.6709492206573486, 0.7089635729789734, -0.11944527924060822, 0.9785808324813843, 1.8592402935028076, 0.052194900810718536, -0.6174919009208679, -0.2713266611099243], [0.9405068159103394, -0.39237362146377563, -1.2152179479599, -0.992999255657196, 2.595831871032715, -0.2635229527950287, -0.7008739709854126, 0.35321059823036194, 0.5933287739753723, 1.4348713159561157, 0.8769404292106628, -0.8231069445610046, -0.491891473531723, -0.24045339226722717, -1.00120210647583, 0.32011643052101135, -0.11264530569314957], [-0.2003888338804245, 0.9682515263557434, -0.17156724631786346, 0.3808475732803345, 1.342499017715454, -0.030524179339408875, -1.5057803392410278, 1.3381569385528564, -0.8391945958137512, 0.4971979558467865, 0.7301830649375916, -2.3517262935638428, 0.025306029245257378, 2.2481179237365723, 1.5197739601135254, -0.13066664338111877, -0.02300449088215828], [1.0988065004348755, 0.29933613538742065, -1.0564926862716675, 0.34533771872520447, -1.2709763050079346, 0.36139532923698425, -0.4598146080970764, 0.18448615074157715, 1.0256288051605225, -0.9518078565597534, -0.23027777671813965, -1.1420128345489502, 0.8659971356391907, -1.1432747840881348, 0.3080444037914276, 0.2216157168149948, -0.8932705521583557], [-0.34876900911331177, 0.5491878986358643, 0.5874794125556946, -0.9811887741088867, 2.2705538272857666, 0.7785844802856445, 0.7507898807525635, 0.12937282025814056, -0.49115753173828125, -0.5491536259651184, -3.2279891967773438, -0.4251909554004669, -0.14322257041931152, -1.832265853881836, 0.7612401843070984, 1.497661828994751, -0.02625878155231476], [0.40518975257873535, -0.6805144548416138, -0.9556745290756226, 0.291677862405777, -0.5460607409477234, -0.001226788037456572, -1.2876968383789062, 0.06693080812692642, 0.851634681224823, 2.3730852603912354, -0.3978264629840851, 0.3998102843761444, 0.6057289838790894, -0.5022013187408447, -1.6092743873596191, -0.9542962312698364, -1.1308610439300537], [-1.8461366891860962, 0.8787152171134949, 1.0017536878585815, -0.9163528680801392, -1.2103619575500488, 0.26008710265159607, 0.4608302414417267, -0.7323423027992249, -0.6220110654830933, -0.6142210364341736, 0.5462873578071594, 0.6229484677314758, -0.6997681856155396, 1.4829665422439575, 0.16438016295433044, -0.3141999840736389, -0.6472253203392029], [-0.5521066784858704, 0.6942983269691467, -0.27993226051330566, -0.2249530553817749, -0.10987775027751923, 2.6485462188720703, -1.1075726747512817, -0.6370732188224792, -1.62552011013031, 0.4963522255420685, 0.8542027473449707, -2.6070339679718018, 1.05595862865448, 0.611518383026123, -0.3176780939102173, -0.09890979528427124, -0.7216545343399048], [0.14061270654201508, -0.8198078870773315, 0.11028939485549927, 0.2609744668006897, -1.1667392253875732, -0.027774257585406303, 1.0478612184524536, -0.3558259606361389, 1.0449591875076294, 0.8665559887886047, 0.6668949723243713, 1.3337180614471436, 0.7371702194213867, -2.0259501934051514, -0.43675243854522705, 0.12483815848827362, -0.5282922983169556], [0.8652744889259338, 0.8653723001480103, 0.05645712837576866, 0.7854648232460022, -0.7051182985305786, 0.40842193365097046, -1.0759245157241821, 0.8825925588607788, 0.11455242335796356, 1.3889552354812622, -1.069952130317688, 0.26333534717559814, -1.090124249458313, -0.3980248272418976, 0.3905962109565735, 0.891818106174469, 0.3411378562450409], [1.2174540758132935, 0.16096295416355133, -0.9471771121025085, 0.21667441725730896, -0.2093924880027771, 1.002127766609192, -1.0186418294906616, -0.5926674604415894, 0.04516559839248657, 0.5771950483322144, -0.656268298625946, -1.4441215991973877, 0.48686590790748596, -0.8049829602241516, 0.14187318086624146, 0.10506600886583328, -0.8889598250389099], [-0.6563236117362976, -0.85996413230896, 0.4282926321029663, 0.09455575793981552, 1.078704595565796, 0.4884471595287323, 0.3317165970802307, -0.4883694350719452, -0.3656098246574402, 1.4294852018356323, -0.2891792356967926, -0.7567331194877625, 0.48646309971809387, -0.7455382943153381, -0.4200182259082794, -0.3420388400554657, 1.364907145500183], [-0.19771312177181244, 0.3527989387512207, -0.07665816694498062, 1.1875768899917603, -0.24437610805034637, 0.16323933005332947, -0.6093524098396301, -0.42357248067855835, 0.8168138265609741, 1.1134275197982788, -0.7005764842033386, 0.5786723494529724, -0.03800258785486221, -1.2280412912368774, 1.645726203918457, -1.3424217700958252, -0.915320098400116], [-0.1794862598180771, -0.4195055365562439, 0.27026569843292236, 0.9494389891624451, -0.7118499279022217, 1.0177942514419556, 1.63685941696167, -0.06174208223819733, -0.12266774475574493, -0.6049255728721619, -1.1650941371917725, 1.110285758972168, 0.7670827507972717, -0.07360298186540604, 0.7390415668487549, 0.025064347311854362, 0.06160534918308258], [-1.3649982213974, 1.7766227722167969, -1.3431802988052368, -0.594146192073822, -0.07833780348300934, 0.14283950626850128, 0.7076846361160278, 0.03412341699004173, 0.3062630593776703, -0.4193367063999176, -1.6543970108032227, -0.8911100029945374, 1.5683001279830933, -0.6629641056060791, -0.7246379256248474, 0.2858453392982483, -0.7447420358657837], [-0.29130110144615173, 0.9426707029342651, -2.018183708190918, -0.9783787727355957, -0.3612312972545624, 0.15229734778404236, -0.4937743842601776, -0.9254803657531738, 0.013438830152153969, 0.2706039845943451, -0.19323576986789703, 0.11309462040662766, 0.11116180568933487, -0.13686086237430573, -0.06653174757957458, 0.0922338217496872, 0.7073885202407837], [0.3714463710784912, -0.2498287856578827, 2.247260093688965, 0.3095155954360962, -0.5321906805038452, -0.797439694404602, 0.18194356560707092, 0.3871929943561554, -1.5105258226394653, 2.545661211013794, 0.6888376474380493, -0.5444144010543823, -0.09011892974376678, 0.9824351668357849, -0.22747071087360382, -0.18003495037555695, 0.6184297800064087], [0.3727235496044159, -0.17868761718273163, -0.6241519451141357, -0.5487583875656128, -0.7396029829978943, 0.05404501035809517, 0.5553243160247803, 0.0031804025638848543, -0.7899074554443359, 0.033443935215473175, -0.8111310601234436, -0.0045515852980315685, 0.7401942610740662, -2.9962453842163086, 0.9083209037780762, 0.19395940005779266, 0.602152407169342], [-1.3338121175765991, 0.4123733639717102, 1.0411628484725952, 0.3284813165664673, -0.34435778856277466, 0.5921852588653564, 0.8370583057403564, 1.0150779485702515, -0.2927425503730774, -1.825393557548523, 1.2993745803833008, 2.599562406539917, -0.9839766025543213, 0.2389964610338211, -0.5933637619018555, 1.6126655340194702, 1.816408395767212], [0.8862341046333313, 1.7305763959884644, 0.7598980665206909, -0.2856481671333313, -0.3211583197116852, -0.16389018297195435, 2.0880513191223145, -0.2765272855758667, 0.12079010903835297, 1.4087830781936646, -0.0428081750869751, 1.3597859144210815, -0.3283940851688385, -0.26157018542289734, 1.836556077003479, 0.11440145969390869, -1.7270578145980835], [-0.9759700894355774, 0.48317140340805054, 0.8729851841926575, -1.4371238946914673, -0.8120636343955994, 0.897655725479126, -0.022422483190894127, -0.3341728448867798, -1.880993127822876, 1.6251170635223389, 0.41890639066696167, -0.32405924797058105, -1.1893210411071777, -1.0372034311294556, 0.5038772225379944, -0.3868299424648285, -0.21700039505958557], [-0.651502788066864, 1.4096789360046387, 0.6045719385147095, -1.4231449365615845, 0.7758257985115051, -0.7805806398391724, 1.8558186292648315, 0.38331741094589233, -0.806542694568634, 0.2746413052082062, -2.3665342330932617, -0.8008392453193665, 1.2828317880630493, 0.5334994196891785, -1.9573360681533813, 1.0798237323760986, 0.15098847448825836], [0.2130974382162094, 0.5287770628929138, -0.7519189715385437, 1.4974921941757202, -0.6333875060081482, 0.2756701111793518, -1.641861915588379, -1.0586961507797241, 0.07015804201364517, 0.5840259790420532, -0.15538907051086426, -0.35066789388656616, -1.2791544198989868, -0.5058704018592834, -0.12511397898197174, -0.08422977477312088, 0.28615400195121765], [0.5907971858978271, -0.40578633546829224, -0.01977546513080597, 2.2165579795837402, -2.1436607837677, 1.7058024406433105, 0.5339215993881226, 2.026303291320801, 0.9183212518692017, -0.30083733797073364, -0.007034501060843468, 0.41614654660224915, -1.3408372402191162, -0.7887991666793823, 0.23779712617397308, 0.8275136947631836, 1.1122852563858032], [0.305015504360199, 0.19370265305042267, -0.5354745984077454, 0.68083256483078, 0.3778862953186035, -0.7811729311943054, 1.5117945671081543, 2.441390037536621, -0.9469699263572693, 0.9714080095291138, 1.263327717781067, 0.15173199772834778, -0.7880758047103882, -0.022176217287778854, 0.24270972609519958, -1.927748680114746, 0.39083540439605713], [1.75639808177948, 1.0585187673568726, -0.3974798917770386, 0.8323156833648682, -0.13971638679504395, 0.9180476069450378, 1.441260576248169, 0.08850383758544922, 0.08849572390317917, 1.4277880191802979, 0.03696831688284874, 0.5660549402236938, 1.8226286172866821, -0.26653003692626953, 0.755342423915863, 0.48754897713661194, -0.11689978092908859], [-0.042673248797655106, -0.022192228585481644, 0.16566863656044006, -1.0631687641143799, 0.8540591597557068, -1.9243327379226685, -0.38409069180488586, 0.5370418429374695, -0.0358438640832901, -0.9745426774024963, 0.2813877761363983, -1.0189985036849976, 0.587541401386261, -0.4103437662124634, 0.06342359632253647, -1.7020317316055298, -1.379313349723816], [0.10930043458938599, 0.4827768802642822, -0.4889698028564453, -0.1390058994293213, 0.7019010186195374, -0.12215789407491684, -1.0960150957107544, -0.8802725076675415, -0.4347986578941345, 0.7777153849601746, 0.5081968903541565, 0.10109709948301315, -0.9393160939216614, 1.0951988697052002, -0.5571823716163635, 0.8402131199836731, 0.13439394533634186], [0.37580186128616333, -0.4892575740814209, -0.27843332290649414, 0.474068820476532, 1.8574607372283936, -0.5277952551841736, 1.3047146797180176, 1.0340244770050049, -0.31472140550613403, 0.6131916642189026, -0.6412410736083984, -1.1953586339950562, -0.9528688788414001, 0.4755711555480957, -0.6916362047195435, -1.2114695310592651, 0.7419441938400269], [0.5512000918388367, 0.80890291929245, -0.37769001722335815, -0.8750078678131104, -0.6122369766235352, 1.4592258930206299, -0.5853080153465271, -0.6543602347373962, 1.1150147914886475, -0.002885720692574978, -0.1284184604883194, -0.6376544237136841, -0.6044142246246338, 0.7166647911071777, -0.5756759643554688, 0.13446156680583954, 1.350701093673706], [1.0877132415771484, -0.762860119342804, 0.3559293746948242, 0.6123015880584717, 0.8025665283203125, -1.743414044380188, -0.9118574261665344, -0.4887266755104065, -1.1072578430175781, 0.2578743100166321, -0.46518099308013916, -0.13440831005573273, -0.8000906109809875, 0.42383337020874023, 2.213679075241089, 1.8470102548599243, 0.12975308299064636], [0.7001566886901855, -0.7712497115135193, 0.574790358543396, -1.7997231483459473, 0.4979984164237976, -0.2892748713493347, 2.2458174228668213, 0.9909952282905579, -0.1322033703327179, -2.17569899559021, -1.2873414754867554, -1.4360600709915161, -0.5890252590179443, 0.4950847029685974, -0.5457354187965393, -1.1385524272918701, 0.9858860969543457], [-1.127569556236267, 0.562606155872345, 1.0427724123001099, -1.6646658182144165, 0.21242882311344147, 0.16101548075675964, -0.23616620898246765, 1.340921401977539, -0.3220435380935669, 2.407059669494629, -0.18309445679187775, -1.1906187534332275, 1.23087477684021, -0.7193844318389893, 0.9356339573860168, 0.512820839881897, -0.26722896099090576], [1.8920189142227173, 0.24949973821640015, -0.0051699052564799786, 1.0297067165374756, 0.1312936693429947, 0.24006253480911255, -0.4874333143234253, -0.7850674986839294, -1.2175005674362183, -0.5629602074623108, -0.35836896300315857, -1.383448839187622, 0.08839993923902512, 0.9328010082244873, 2.204172372817993, -0.179134339094162, 0.0710306391119957], [-0.3587018847465515, 1.7080885171890259, -0.2829521894454956, -2.0117785930633545, 0.8229350447654724, -1.197721242904663, -0.19115208089351654, 1.450566053390503, 0.14769111573696136, 1.1600004434585571, 0.3134866952896118, 1.1848185062408447, 1.2841352224349976, -0.06718537211418152, -1.1303675174713135, -1.8262419700622559, -0.6642305850982666], [-0.15158869326114655, -0.274724543094635, -0.721971869468689, 1.0595513582229614, 1.0361461639404297, 0.2303958535194397, 0.0875457152724266, 0.790094256401062, 0.6975026726722717, 0.8916664123535156, -0.21816851198673248, -0.27599990367889404, 0.1066581979393959, -0.6528334617614746, 0.24841755628585815, 1.052931308746338, -1.5841792821884155], [-0.5007488131523132, 0.27992069721221924, -1.5131337642669678, 0.7601005434989929, 1.410032033920288, 1.0015677213668823, 0.6531873941421509, 1.743433952331543, -1.54915452003479, 1.6293984651565552, -0.060015928000211716, 0.36049386858940125, -1.4318703413009644, 1.1010149717330933, 0.9006044268608093, 0.6028022766113281, -0.8119040727615356], [1.2822803258895874, -0.6352537870407104, 0.5870482325553894, -1.273559808731079, 0.9516103863716125, -0.8198186755180359, 0.7749704122543335, -1.4266403913497925, 0.7439283728599548, -1.2563605308532715, 0.2788006663322449, -0.3578689396381378, -0.5151386260986328, 1.6985645294189453, -0.187247171998024, -0.6488054990768433, -0.2583421468734741], [0.8238169550895691, -0.5243200063705444, 0.039078518748283386, -0.8090701103210449, -1.2260782718658447, -1.5559301376342773, 0.0214046873152256, 0.8258307576179504, 1.0110177993774414, -1.797366976737976, 0.7407795786857605, -0.5509389042854309, -0.3101235330104828, -1.8015031814575195, -0.5252248644828796, 0.9974351525306702, -0.18824328482151031], [0.0854257270693779, -1.0495688915252686, -0.20350752770900726, -0.4989195466041565, -0.4091338515281677, -0.20904168486595154, -0.1096617728471756, -1.0859565734863281, 1.2723218202590942, -0.012922422960400581, 1.2112500667572021, -0.4481644034385681, 1.7027965784072876, -0.1584809124469757, 0.5055519342422485, 0.13871701061725616, 1.0505064725875854], [1.4829884767532349, -0.9895345568656921, -0.26060283184051514, -0.6114627718925476, 1.1901365518569946, 0.202562615275383, 0.8745547533035278, -0.7603468894958496, -0.19811372458934784, 0.11209075897932053, 1.8061121702194214, 0.5955353379249573, 1.1959028244018555, 2.2661659717559814, 1.147387981414795, -1.2214457988739014, 0.03553006425499916], [0.9567302465438843, -1.0266828536987305, 0.7664517164230347, 1.2630800008773804, -0.7049150466918945, -0.7661646008491516, -1.8739854097366333, 0.21462999284267426, 1.0360759496688843, 0.1830318123102188, -1.2680175304412842, 0.1202433779835701, 1.1696994304656982, -0.0888902023434639, 0.019383642822504044, 0.5651543140411377, -0.37766847014427185], [-0.3673706352710724, -0.07833941280841827, 0.019947275519371033, 1.4302630424499512, -1.0443696975708008, 0.7809295058250427, -1.973388671875, -0.01895112171769142, 1.6374708414077759, -1.1036174297332764, 1.3259450197219849, 2.3541619777679443, 1.7322888374328613, 0.642325222492218, -2.0115389823913574, -1.1005423069000244, 1.3380086421966553], [0.24049586057662964, -0.578171968460083, -1.4056341648101807, -0.758333146572113, -1.1807467937469482, -0.5136145353317261, 0.813240647315979, 0.31129151582717896, 0.6550722718238831, -0.24523620307445526, 0.7278391122817993, 0.6804831624031067, -0.089222751557827, 0.516254723072052, -1.7207026481628418, 1.2855108976364136, -0.21990720927715302], [-1.035117506980896, 2.4241318702697754, 0.4314095079898834, 1.2873224020004272, 0.7982743382453918, -1.1483476161956787, 0.8857823014259338, 1.117714285850525, -0.7612211108207703, 0.24489089846611023, -0.9914901852607727, 0.7384805679321289, -1.3549473285675049, -0.5733028650283813, 0.3263399600982666, 0.7764123678207397, -0.5975291728973389], [2.037982702255249, 0.954063892364502, 0.5439779758453369, -0.11275029927492142, 0.058153606951236725, 0.1013590469956398, 0.6195468306541443, 1.6750189065933228, 0.1493072360754013, -0.6737238764762878, 0.6929890513420105, -0.6684750318527222, -0.8943246603012085, -0.1966976374387741, -0.8833022713661194, 1.7501190900802612, 1.4800121784210205], [-0.8261780738830566, -0.9295265674591064, -0.3900270164012909, 0.6391775608062744, -0.4626944363117218, -0.02272355556488037, -1.3003261089324951, -1.690779447555542, 1.2048298120498657, -1.7920628786087036, 1.7080706357955933, -0.6095041036605835, -0.9524140357971191, 1.2677785158157349, -1.5508127212524414, 1.1012259721755981, 0.37881240248680115], [-2.2179105281829834, -1.1335722208023071, -0.3922789692878723, -1.3085856437683105, 0.061346061527729034, -0.6832478642463684, 0.38824141025543213, -1.1016358137130737, 0.13271357119083405, 1.491713523864746, 0.008980256505310535, 1.223671555519104, 1.292562484741211, 0.7911466360092163, 0.005850051995366812, 1.161062240600586, -0.8164374232292175], [-0.1587027758359909, 2.518418312072754, -0.45290425419807434, -0.48629704117774963, -0.2365427315235138, 0.8065325617790222, -1.1594825983047485, -0.08841407299041748, -1.5477516651153564, 1.0452407598495483, 1.042022943496704, 0.04508955776691437, -0.13853484392166138, -0.11876054108142853, 2.050558567047119, 0.656074047088623, -0.8843252658843994], [-1.0974016189575195, -0.015188677236437798, -0.23056715726852417, -0.45515987277030945, 1.1559875011444092, -0.48617514967918396, -1.6963629722595215, 0.25073733925819397, 0.42286574840545654, -0.46657124161720276, 1.5254533290863037, -0.8336842656135559, -0.5386005640029907, -0.6213231682777405, -0.887226402759552, -3.935267448425293, -1.5787427425384521], [-1.1499842405319214, -0.3055868148803711, -0.5865346193313599, -0.8917707204818726, 0.9981303811073303, 0.08425046503543854, -2.3648126125335693, -0.5592081546783447, -0.6747546792030334, 0.8671072125434875, 1.2856436967849731, 0.05152374505996704, -0.2975984513759613, 0.8717706799507141, -0.3754083514213562, 0.4228188693523407, 1.595442295074463], [0.9722764492034912, -0.8172664046287537, 0.6780886650085449, 0.7256507277488708, 0.32227209210395813, -0.3723350465297699, 0.49238359928131104, 0.15072308480739594, 1.6689190864562988, 0.0827728658914566, 0.07733030617237091, -0.33418968319892883, 0.8606722354888916, -0.7219854593276978, -0.49123623967170715, 0.45540881156921387, 2.0542094707489014], [0.9464825391769409, 0.706468403339386, -0.07475051283836365, 0.9477550983428955, -1.2473232746124268, -0.06518157571554184, 1.9560619592666626, 0.11176840960979462, -0.6145145893096924, -0.15278366208076477, -2.5774121284484863, 0.07801882922649384, 1.0567961931228638, 0.6949378848075867, 0.27251794934272766, 0.8842623233795166, 1.689605951309204], [1.4046810865402222, -0.1529795229434967, -0.23481349647045135, -1.9379767179489136, 2.0221080780029297, 1.1885205507278442, -0.6763437390327454, 1.2979118824005127, -0.4277331531047821, -0.3244583010673523, 0.3564491868019104, 0.7826249599456787, 0.2929384112358093, 0.10094117373228073, -0.4911307096481323, 2.9322729110717773, 0.020301979035139084], [0.7605171203613281, -0.7999973297119141, -3.030992031097412, 0.20987239480018616, 0.569930374622345, -0.14277125895023346, 0.7262643575668335, -0.6348190903663635, -0.1856776624917984, -1.461859941482544, 1.3622808456420898, 2.388340950012207, 0.9686297178268433, 1.2865076065063477, 0.7853754162788391, 0.49367907643318176, -0.12868818640708923], [-0.16625121235847473, -1.1391816139221191, -1.1220242977142334, 2.003148317337036, 0.6679829359054565, -0.7558732628822327, -0.6056153774261475, -2.410313844680786, -0.08337382227182388, 0.790489912033081, -0.3024379014968872, -1.036664605140686, -0.2838365435600281, -0.8996080756187439, -0.8632701635360718, 0.4131081700325012, 0.34084558486938477], [0.7139325737953186, 0.8722613453865051, 0.8613048791885376, -1.5631905794143677, 1.8249945640563965, -0.41307777166366577, 0.5317074060440063, 0.5832005143165588, -0.9559326767921448, 0.2245711386203766, -1.4571685791015625, -0.4653349220752716, -0.6229854226112366, 0.22155877947807312, 1.6053541898727417, 0.0928562581539154, -1.0670150518417358], [1.338706374168396, 0.6205415725708008, 0.16998140513896942, -0.45274490118026733, 1.732845664024353, -0.7482432126998901, -0.0030680259224027395, 1.0774991512298584, -0.5030921101570129, -0.6752488613128662, -0.9778104424476624, -2.416966676712036, -2.0880424976348877, 0.02229297161102295, -0.35890623927116394, 2.9420697689056396, 0.6585450172424316], [-1.4521119594573975, 0.20286747813224792, -0.13246645033359528, -0.1072903722524643, -1.4318937063217163, 1.229637861251831, -0.40193334221839905, -0.7218157649040222, -1.637549638748169, -0.0733916163444519, 1.144433617591858, 1.3651719093322754, -0.7560766935348511, 0.18640351295471191, 2.326890707015991, 0.1641424596309662, 1.2312648296356201], [-1.0171430110931396, 0.6013021469116211, 0.36964884400367737, 2.344670534133911, -0.4291982352733612, -1.2424285411834717, 0.1713184267282486, 0.1985560655593872, 0.15552321076393127, -0.29597437381744385, -0.41269975900650024, -1.2522801160812378, 1.554256796836853, -0.983925461769104, -0.9648517370223999, 0.044668860733509064, -0.29543548822402954], [1.098471999168396, 1.6488970518112183, -1.795620322227478, 0.5689619183540344, -0.8766728043556213, -0.20813435316085815, 0.6332639455795288, -1.2533292770385742, -2.1319632530212402, -1.0126579999923706, 0.5620248913764954, 0.3850523829460144, 0.846703290939331, 0.9380434155464172, 2.5030999183654785, 1.0947613716125488, -0.09526286274194717], [0.1679040938615799, 0.8970690369606018, 0.5757666230201721, 1.3989596366882324, -0.22202448546886444, -0.9821462035179138, -1.6619436740875244, 1.936545491218567, 0.6149886846542358, -2.663618326187134, 1.2875125408172607, 2.43829607963562, -1.2260326147079468, 1.14015531539917, -0.07887892425060272, -0.7500450611114502, -1.5112369060516357], [0.4047791063785553, -1.1286908388137817, -0.0934814065694809, -0.7033395767211914, -0.5742884874343872, 0.1864064782857895, 0.9528263211250305, 0.31603044271469116, 1.4729987382888794, -0.48456522822380066, -0.6049508452415466, -1.5939295291900635, -1.1758301258087158, -1.27982759475708, -0.27281731367111206, 0.8954700231552124, -0.9450894594192505], [0.6911397576332092, -0.4243929386138916, -0.6471568942070007, -2.0792319774627686, 0.0787096694111824, 0.8996862769126892, 1.2187942266464233, 1.0606056451797485, -0.8303619623184204, -0.029377805069088936, 0.8379238843917847, 0.23119722306728363, -0.7684587836265564, 0.45534181594848633, 1.3072243928909302, -0.3501903712749481, -2.197946786880493], [0.3154323697090149, 0.1000397577881813, -0.2775326073169708, 1.9126850366592407, 0.8170425295829773, -0.029960351064801216, -0.3508298695087433, -0.0959947258234024, 0.40479642152786255, 0.7617528438568115, -1.1733776330947876, -1.478399634361267, -1.6900782585144043, 0.6022710204124451, -0.2576392889022827, -0.1370794028043747, -0.07849565893411636], [-0.17461705207824707, 0.8921324014663696, -1.9398422241210938, -0.6487495303153992, -1.9617786407470703, 0.9184712767601013, 0.23811949789524078, -0.16859476268291473, 1.3793801069259644, -0.0948486328125, -0.365169882774353, -0.6712799072265625, 0.6957370638847351, -0.012158366851508617, 1.225674033164978, 0.820278525352478, -0.39784300327301025], [-1.8992263078689575, 1.0723130702972412, 1.9067292213439941, -0.5406785607337952, -0.885033905506134, 1.222407341003418, 1.7107068300247192, -2.4764914512634277, -0.8920329809188843, -0.8118414878845215, -1.3849408626556396, 0.42753031849861145, -1.92261803150177, 0.32024598121643066, -0.9967604875564575, -1.3387424945831299, -1.0881876945495605], [-1.486072301864624, -0.15147268772125244, 0.7783825397491455, -0.3977695107460022, -0.5630879998207092, -0.575768232345581, -1.5348987579345703, 0.567338764667511, 0.18938305974006653, 1.0428701639175415, 0.3613690137863159, 0.37725064158439636, 1.4608873128890991, 1.3441709280014038, 0.9885611534118652, 0.40733230113983154, -1.6426714658737183], [-1.239888310432434, -0.5299818515777588, -1.2619290351867676, -0.02477746270596981, -0.24052143096923828, 1.2095179557800293, 1.805616021156311, 0.36185792088508606, -1.5070607662200928, -0.5562894940376282, 1.0618999004364014, -0.643012285232544, 1.5078189373016357, 0.28746578097343445, -1.485369324684143, 0.4568909704685211, -0.9504270553588867], [-0.037438973784446716, -0.32204848527908325, -1.1565135717391968, 1.5849101543426514, 1.4472379684448242, 0.07493583112955093, -0.7366694808006287, 0.7031174302101135, -1.6723406314849854, -0.95440274477005, -0.9663767218589783, -1.1542388200759888, -0.1575218141078949, 1.8432700634002686, 1.476720929145813, -0.6509607434272766, 1.0615735054016113], [-0.6971783638000488, -0.1609373390674591, 0.038644418120384216, -1.0995352268218994, -0.585225522518158, -0.29058194160461426, 0.40434542298316956, -0.6821398735046387, 2.6184043884277344, 0.8297943472862244, 0.006573102902621031, 0.654721736907959, -0.5438902974128723, -0.4760746657848358, -0.6059413552284241, -0.8128408789634705, 0.15193350613117218], [-0.964982807636261, -0.993811309337616, -1.1312474012374878, 0.4526609182357788, -0.9175252914428711, 0.2851656675338745, 0.2668100893497467, -0.36259064078330994, -0.17495709657669067, 0.24329976737499237, 0.3255300223827362, 0.7969179153442383, 0.814415693283081, 0.8259263038635254, -0.13727064430713654, -0.35912561416625977, 0.5889396071434021], [-0.37116488814353943, -3.438666820526123, -0.8029970526695251, 1.2820464372634888, -0.04016946628689766, -1.9645769596099854, -0.980791449546814, 0.5555395483970642, 0.26574382185935974, 1.128261923789978, -1.0210719108581543, 0.031679388135671616, 0.48528462648391724, -0.47638240456581116, 1.0870777368545532, 0.4161664843559265, 0.2505510151386261], [-0.2818623185157776, 0.3570398986339569, 0.09478146582841873, -0.21937154233455658, -0.4871048927307129, -1.835048794746399, -0.4727839231491089, -0.8147858381271362, 0.3094673752784729, 0.03227570280432701, 1.8125420808792114, -0.5501888394355774, 0.5442233085632324, -1.3545262813568115, 1.0103061199188232, -1.6799044609069824, 0.31544652581214905], [-0.2025119662284851, 0.09717442840337753, 1.0615202188491821, 0.15682058036327362, -0.06871199607849121, -1.0010342597961426, 1.6211720705032349, 1.4558953046798706, -0.30499324202537537, 0.8351302146911621, 0.32996317744255066, 0.027421515434980392, -1.1700634956359863, -1.265391230583191, -1.4990736246109009, -0.3066157400608063, -0.8393967747688293], [-0.33006393909454346, 1.1935763359069824, 2.7768468856811523, 0.602517306804657, -0.06659062206745148, -0.6173122525215149, 1.3728430271148682, -1.7966735363006592, -0.49570417404174805, -0.4442955553531647, 1.0171135663986206, 0.7546589970588684, 0.11273116618394852, 0.044506147503852844, 0.48844650387763977, -0.7588611245155334, 0.037547629326581955], [1.4231101274490356, -1.335505485534668, 1.3420734405517578, 0.08665365725755692, 0.31530502438545227, -0.48877793550491333, -0.7800324559211731, 1.7610104084014893, 0.7551772594451904, -1.0023242235183716, 0.5430467128753662, -0.6037865281105042, -0.09639164060354233, 0.2505963444709778, -0.49802854657173157, -1.062996745109558, 0.36436551809310913], [1.1679385900497437, -1.0229134559631348, 0.032258354127407074, -0.39360690116882324, 0.9495433568954468, -0.29406502842903137, -0.9653860926628113, 1.1417646408081055, -0.17602601647377014, -0.24198883771896362, 0.9590027332305908, 0.15301565825939178, -0.9366037249565125, -0.20934711396694183, -0.028136469423770905, -3.337754964828491, -1.4486757516860962], [0.01803772710263729, 1.3212697505950928, -0.0847821980714798, 1.1499474048614502, 0.8926532864570618, -0.21735483407974243, -0.3175991177558899, 0.19546666741371155, -0.49864283204078674, -0.4465256929397583, 0.519590437412262, -0.439498633146286, 0.7431873083114624, -0.5976791977882385, 0.4387192130088806, -0.10515867173671722, 0.8277611136436462], [0.6164135336875916, 1.3002458810806274, 1.8586714267730713, 0.2668668031692505, -1.071978211402893, 0.22602243721485138, 1.2686671018600464, 1.8406850099563599, 0.27824756503105164, 0.0007688783225603402, -1.6671943664550781, -0.16711081564426422, 0.3550630509853363, -0.9576869010925293, -1.772464394569397, 0.16934190690517426, 1.6477731466293335], [1.6923733949661255, -0.43082794547080994, 0.2681597173213959, 1.6763932704925537, 1.1214224100112915, 0.39883628487586975, 0.8188270926475525, 1.6014050245285034, -0.003761876840144396, -0.5234348773956299, 0.5294883251190186, 0.5817199349403381, -0.6440669298171997, -0.7503964304924011, -1.5167258977890015, -0.07453292608261108, 0.8348323702812195], [-0.1937105357646942, 1.3745423555374146, 0.4644404649734497, -0.00019849267846439034, 1.2508231401443481, 0.3167805075645447, 2.0022716522216797, -1.185850977897644, -0.5610218644142151, 0.25016894936561584, -0.24601314961910248, -0.9985230565071106, -1.1981574296951294, 0.5573112368583679, -0.7838326692581177, -0.9930891394615173, 1.7804911136627197], [2.2813217639923096, -1.1565510034561157, -0.4189056158065796, 1.020124077796936, -0.4517107605934143, 1.505210518836975, -0.9598385095596313, -1.894681453704834, 0.13983400166034698, -0.39348816871643066, 0.9070799946784973, 0.2751747965812683, 0.22828610241413116, 1.7461272478103638, 0.2615911662578583, -0.731342077255249, 0.526852548122406], [-0.9784111380577087, -0.8552486896514893, 0.8711716532707214, -0.3034019470214844, 1.0606517791748047, 0.3380734622478485, 0.7700921893119812, -0.5798751711845398, 0.40362998843193054, 0.8309981226921082, -0.2668207287788391, 1.2771766185760498, -1.035832405090332, -1.1539366245269775, -0.653659462928772, -0.22572974860668182, 1.1473222970962524], [0.16704341769218445, -0.468475341796875, 0.8589558005332947, -0.4741485118865967, -1.4740194082260132, -0.003400631947442889, 0.8223868012428284, 0.03729601576924324, -1.2311331033706665, 1.7634515762329102, -0.47910207509994507, -1.4045486450195312, -2.0614542961120605, 0.8550735116004944, 1.214460015296936, 0.026059651747345924, 0.21008194983005524], [-0.2442772537469864, -1.093508243560791, 0.9373008608818054, -0.2689087688922882, -0.2555569112300873, 0.6589683294296265, 0.2834521234035492, 0.44671630859375, 0.9334922432899475, -1.479870080947876, 0.155010387301445, -0.6424159407615662, -0.213538259267807, 0.4225933253765106, -0.2068597823381424, 0.03239155188202858, -0.9513800144195557], [0.31871888041496277, 0.6655932068824768, -0.6428605318069458, 0.8602855801582336, -0.13105809688568115, -0.4132002890110016, 0.39270883798599243, -1.375234603881836, 0.26581335067749023, -1.5069621801376343, 0.8349512219429016, 0.9731433987617493, -0.5580927729606628, -0.4839867055416107, -0.5005508661270142, -0.7500121593475342, -1.075526237487793], [-1.0288554430007935, 0.660927951335907, -2.070826768875122, -0.277485191822052, 0.06591533869504929, 1.5813355445861816, -0.7702348828315735, 0.49443596601486206, -0.3932957053184509, -0.22388385236263275, -0.9644606113433838, 1.9312080144882202, -1.8646175861358643, 1.1478900909423828, 0.38719579577445984, 1.6333942413330078, -0.449861615896225], [2.65730357170105, 0.10621081292629242, -0.8788564801216125, -0.7474650740623474, -0.2145458310842514, 0.025977274402976036, -0.14632219076156616, -0.5842387676239014, -1.6942837238311768, 0.7945108413696289, 0.019709840416908264, -0.44248849153518677, 0.49209630489349365, -0.8589385747909546, 1.2588293552398682, -0.19018681347370148, 1.0422652959823608], [-0.49228256940841675, -0.9121019840240479, 1.4569658041000366, -0.7291790246963501, -0.2832516133785248, 0.5437008142471313, 1.094260334968567, -0.784428060054779, 0.8490743041038513, 0.5674183368682861, 0.030259812250733376, -1.5315961837768555, 1.5892633199691772, 0.7014524340629578, -0.02711174078285694, 0.8908843398094177, -0.2846684753894806], [-0.9147483110427856, -0.29653918743133545, -0.7307271361351013, 0.44888946413993835, -0.20851142704486847, 0.569141149520874, -0.7035759091377258, -0.45493555068969727, -0.5869743227958679, 0.004918505437672138, 0.15311197936534882, -0.6861391663551331, -1.5858722925186157, 0.1100713461637497, 0.5889512896537781, -1.1403154134750366, -0.7536285519599915], [0.7907223105430603, 0.3566535413265228, -0.45263999700546265, -0.7332708835601807, -0.23152399063110352, -0.6523203253746033, -0.3929585814476013, -0.7485275864601135, 2.030599355697632, 0.683314859867096, 0.35778170824050903, 0.46148863434791565, 0.4833615720272064, 0.1836152821779251, 0.8148818612098694, 1.629582405090332, -0.9796536564826965], [-1.0938550233840942, 0.05869346484541893, -0.13854992389678955, 2.0125396251678467, -1.2533279657363892, -1.5404380559921265, 0.49917325377464294, -1.1644188165664673, -1.3676421642303467, 0.8188522458076477, 0.4091216027736664, 0.8837682008743286, 0.9954326152801514, 0.04492561146616936, 0.15207836031913757, 0.30781862139701843, 1.0682353973388672], [-0.700516402721405, -0.07641305029392242, -1.1865426301956177, -0.3286695182323456, -0.8346642851829529, 0.17310582101345062, -1.7124334573745728, -0.299535870552063, -0.1594415307044983, 0.3361049294471741, -1.8127000331878662, 0.7631425857543945, 0.058485422283411026, -0.46987491846084595, -0.4413309395313263, -1.0164027214050293, -1.8407117128372192], [-0.6567986607551575, 0.9638752341270447, 0.7557602524757385, 2.7031257152557373, -1.4722933769226074, -1.4049934148788452, -0.08963270485401154, 0.30016419291496277, 1.2685112953186035, 0.7450052499771118, -0.8159964084625244, 1.9270853996276855, -0.4156985282897949, -1.2704311609268188, 0.3575076162815094, 0.6024227738380432, 0.809834897518158], [0.8872011303901672, 0.05142604932188988, 0.006329780910164118, 1.5630208253860474, -1.8341439962387085, 1.628265619277954, 0.431822270154953, 0.268155038356781, -2.193223237991333, 0.03175705298781395, -0.6338439583778381, -0.9608768224716187, -2.1379590034484863, -0.43718570470809937, 0.43897339701652527, 0.04284103214740753, 0.28292611241340637], [-0.23652881383895874, 0.8146721720695496, -1.5459383726119995, 0.8251770734786987, -0.29653409123420715, 0.9271449446678162, -1.093420147895813, -0.5339037179946899, 2.529680013656616, -0.7903551459312439, -2.7484207153320312, 1.6749955415725708, 1.2562143802642822, -0.6599418520927429, -1.0582114458084106, -0.25932613015174866, -0.6022806167602539], [1.6459952592849731, 0.13257648050785065, -0.6065828800201416, -1.4596058130264282, 0.49296244978904724, 0.16039226949214935, 1.8159345388412476, -0.5839798450469971, 1.325205683708191, -1.1291084289550781, 0.637874960899353, -1.5146859884262085, 0.2417847067117691, 0.338511198759079, -0.10500722378492355, 0.6485213041305542, 0.4712349474430084], [0.6881434917449951, 0.6366447806358337, 0.48249194025993347, 0.6086957454681396, -0.12934792041778564, -1.1006839275360107, -0.2257242649793625, -1.2441747188568115, 1.9381941556930542, -0.38822367787361145, -0.9911811947822571, 0.0007541723898611963, -0.2690662443637848, 1.5608997344970703, -1.342576503753662, 1.3916481733322144, -0.02836541272699833], [-0.5368707776069641, 0.6311377286911011, -0.8250714540481567, -0.8323097229003906, -0.261786550283432, 0.7212520241737366, -0.8347898721694946, -2.0619630813598633, -1.2817219495773315, -0.4936600625514984, -1.0038998126983643, -0.2833813428878784, 0.9554198980331421, 1.6527011394500732, -0.870808482170105, -0.9485686421394348, -1.6383517980575562], [-1.2736440896987915, 0.4733888506889343, -0.8163003921508789, 1.186931848526001, -0.3592780828475952, 0.71540367603302, -0.10145679116249084, 0.43718382716178894, -0.9587103724479675, -0.30054086446762085, -0.6691474318504333, -1.7375239133834839, -0.0177801251411438, 1.9958820343017578, -1.2655280828475952, -1.0983136892318726, 1.2068817615509033], [-1.331528902053833, 0.5860422253608704, -1.0774378776550293, 0.37786099314689636, -1.5498764514923096, -0.5217664837837219, -0.27592816948890686, 1.006719708442688, -0.04750822111964226, -0.07698861509561539, 0.986612856388092, -0.16922925412654877, 0.6147804856300354, 1.3408557176589966, -0.9843992590904236, -2.1984918117523193, 0.41533738374710083], [0.594167172908783, -1.412781000137329, 0.03818894922733307, 0.8993695378303528, -0.27623069286346436, 1.9414750337600708, -0.618642270565033, -0.5137022137641907, -0.6958655714988708, -0.8643026351928711, 0.6557245850563049, -0.19353032112121582, -0.5437331199645996, 1.7334418296813965, -1.0017000436782837, 0.40153631567955017, 0.4566708505153656], [0.2418893426656723, -1.1066988706588745, 0.3139680027961731, 0.8959828615188599, 0.32644927501678467, 1.3408485651016235, 0.30167144536972046, -1.9562090635299683, -1.5292773246765137, -0.325147420167923, 0.755401611328125, 2.1925735473632812, 1.5773411989212036, -0.43503719568252563, 0.7499603033065796, -1.0744504928588867, 0.7531412243843079], [-1.9838135242462158, 1.215433120727539, -1.8243026733398438, -1.0995891094207764, -0.3350651264190674, -1.1455426216125488, -0.9510628581047058, -0.37644845247268677, 0.38542768359184265, -1.248397707939148, 0.012992586009204388, -0.2201281636953354, 0.05239647999405861, -0.6032021045684814, 0.08226064592599869, 0.5934121608734131, 1.7157009840011597], [-0.36438724398612976, -0.9616403579711914, -1.1087067127227783, 2.146287441253662, -1.3762547969818115, -0.9041711688041687, -0.4767792224884033, 0.9768732190132141, 0.43910983204841614, -1.216322660446167, 0.012332535348832607, -0.22369007766246796, -0.35099172592163086, -1.2509857416152954, 0.7664265632629395, 1.2265721559524536, 0.1620444804430008], [0.9583240151405334, -1.645289421081543, -0.3668593168258667, -0.2238212376832962, 0.5300773978233337, 0.10436175018548965, 0.9255507588386536, 0.8616661429405212, 0.7955017685890198, -0.48529574275016785, 0.6582739949226379, 0.26608550548553467, -3.540238380432129, 0.4545789659023285, -0.36242711544036865, -0.7307863235473633, -0.10960882902145386], [1.1315752267837524, 0.589412271976471, 0.3837960958480835, 0.6075608730316162, 0.6480244994163513, -0.6147425770759583, 0.32771456241607666, -1.8481684923171997, 0.5797683000564575, -0.6236211657524109, -1.329283356666565, 0.5118403434753418, 1.0598855018615723, -1.4659758806228638, -0.24158549308776855, 1.8062553405761719, 1.8804490566253662], [-0.6296392679214478, -1.246910572052002, 0.42018115520477295, -0.80185467004776, 0.07205236703157425, 1.6545311212539673, 1.311131477355957, 0.7169901132583618, 0.5271096229553223, 0.6980366110801697, -0.7386956810951233, -0.7098803520202637, -0.18284021317958832, -0.1671069860458374, -1.373943567276001, -0.6370531916618347, 0.4598301649093628], [0.9122350811958313, 0.1337282806634903, -0.8672018051147461, 0.14915649592876434, 0.25975751876831055, -0.08356261253356934, 1.198301911354065, -0.6618455052375793, 0.6984452605247498, 0.4804247319698334, -0.5734187364578247, 0.5628589391708374, -0.8141213655471802, -1.652333378791809, 0.7131617069244385, -0.6030683517456055, 0.2550964057445526], [-0.6415136456489563, 0.6459039449691772, -1.1853704452514648, 0.07292278110980988, 0.7944018840789795, -0.002839513588696718, 2.0230143070220947, 2.6310977935791016, -0.5116825699806213, 0.17125201225280762, 1.1187260150909424, -1.6065778732299805, -0.664804220199585, -0.29296624660491943, -1.3246151208877563, 0.6755207777023315, -0.8297014832496643], [-0.35584768652915955, 0.2536117136478424, -0.7450721859931946, -1.0353106260299683, -1.213362216949463, -1.7683132886886597, 0.5140033960342407, 0.5821985006332397, 0.5704352855682373, -1.9068251848220825, 0.7621946930885315, -0.616150975227356, -0.6971518397331238, -1.0810332298278809, -0.3166252374649048, -1.731034755706787, 0.22997817397117615], [0.04444362595677376, -0.9486108422279358, -0.32592475414276123, 0.5790333151817322, 0.3206245005130768, -0.05769938975572586, 0.31751248240470886, 0.7995282411575317, -0.7351248264312744, 1.3741930723190308, 1.2326451539993286, -0.5024372935295105, -0.7633635997772217, -0.9203546047210693, 1.2234888076782227, -0.2108747363090515, -1.0107824802398682], [-0.9492944478988647, 0.32305824756622314, -0.7455175518989563, 0.9457876682281494, 0.19351333379745483, 1.0792335271835327, 1.167620062828064, -0.6536726951599121, -1.2944163084030151, 0.12025745958089828, 0.11081624776124954, -1.799888014793396, 1.5913186073303223, 1.0450347661972046, 0.7053442597389221, 0.18481621146202087, -0.9061198830604553], [-0.933329164981842, -0.9124760627746582, 0.6041128635406494, 1.4526201486587524, 1.3801178932189941, -0.45097100734710693, 0.6909180283546448, -0.5676726698875427, -0.35965871810913086, 0.6900132298469543, 0.2694864273071289, 1.5674443244934082, -0.9730345010757446, 0.9360951781272888, -0.7706479430198669, 1.157281517982483, 0.7177503108978271], [0.057548362761735916, -0.4702172875404358, -0.28167831897735596, 0.5424917936325073, 1.0153647661209106, -1.378444790840149, 0.4354901909828186, 0.3128618597984314, 0.11139388382434845, 0.028884829953312874, -0.319087415933609, 1.0449267625808716, 0.7945014238357544, 0.13714507222175598, 1.4428459405899048, -0.19772283732891083, -0.9128961563110352], [-0.6287983655929565, -0.16352085769176483, -0.5468926429748535, 2.700308322906494, 1.0189327001571655, -1.5187904834747314, 0.6905075311660767, 0.5664662718772888, 1.1529823541641235, 0.018052276223897934, -0.08772478252649307, 0.2873581051826477, 0.4363713562488556, 0.10164433717727661, 0.016156241297721863, -0.755774199962616, 0.9932541847229004], [1.251904845237732, 0.6882965564727783, -0.05367070809006691, 1.7693746089935303, 0.7144219875335693, -1.123539924621582, 0.7086023092269897, -0.6366153359413147, -1.2711328268051147, 0.8026304244995117, 1.0807265043258667, 1.1554973125457764, -0.38148069381713867, -0.6165456175804138, 1.9817918539047241, 2.238382577896118, -2.738523483276367], [-1.1208666563034058, -0.6438554525375366, 0.16089186072349548, -0.24189729988574982, 0.605225145816803, -0.7776435017585754, 1.4668296575546265, -0.12181863933801651, 0.665220320224762, -0.008909227326512337, 0.13522519171237946, -3.124478340148926, -0.7242644429206848, 1.3410741090774536, -1.0934107303619385, -0.25763002038002014, -0.6358357071876526], [0.31111979484558105, -1.0836279392242432, 0.38307347893714905, 0.5550445914268494, -1.105518102645874, 1.960606336593628, -0.9364646673202515, -1.3547472953796387, -1.3256908655166626, 0.20965822041034698, 0.2820379137992859, -0.4708791971206665, 1.3976786136627197, 0.7208226323127747, 0.10675782710313797, 0.20498932898044586, -1.0212477445602417], [2.0925252437591553, 1.719499945640564, -1.3837467432022095, 0.25955459475517273, -0.29474589228630066, -1.1190671920776367, 0.13214316964149475, -0.8433415293693542, -0.30010148882865906, 0.648640513420105, -0.7557039856910706, -0.3754631280899048, -0.9852792620658875, 0.9863196015357971, 0.8591579794883728, -0.17367422580718994, -0.9361183643341064], [0.9708389639854431, -0.9926674962043762, 0.7699040770530701, 1.3264862298965454, 0.25689631700515747, 0.5530962944030762, -0.03089669719338417, -1.3722513914108276, 1.211275577545166, -1.2374789714813232, -0.26542311906814575, -0.8579416275024414, -0.1323518455028534, -0.07186378538608551, 0.5500611662864685, 1.0388941764831543, 0.925096333026886], [0.6675909757614136, 1.031905174255371, -0.8405376076698303, -0.24130405485630035, -0.8555310368537903, -0.2864176034927368, -0.6832482218742371, -1.3796058893203735, -0.9505475163459778, 1.9754958152770996, -1.0549646615982056, -0.7714637517929077, 0.2471025437116623, 0.03051220253109932, -2.6780171394348145, 0.760121762752533, -0.6297019720077515], [-1.1097673177719116, -0.09288804978132248, 0.02323930710554123, 1.5844511985778809, -0.4545799195766449, -0.5406355857849121, 1.6158862113952637, -0.17793700098991394, -0.1705496609210968, 0.5500285029411316, 0.24097368121147156, 1.1138405799865723, 0.13955067098140717, 0.7691506743431091, -0.4836216866970062, -0.3113054931163788, -0.8510782718658447], [-0.7601413130760193, -1.9342749118804932, -0.5811247825622559, -0.6580162644386292, -0.34890982508659363, -0.9821637868881226, 0.19624675810337067, 0.7378028035163879, 0.17200028896331787, -0.4941807687282562, -1.0375239849090576, -0.05172347649931908, -1.2840603590011597, 0.5501308441162109, -0.4191947877407074, -0.5253615379333496, 0.14249356091022491], [-1.076829195022583, 0.9491019248962402, 0.17376674711704254, 0.5028396844863892, -0.3170900046825409, -0.11434595286846161, -0.20116925239562988, 0.5583582520484924, 0.4274098575115204, 0.8585482239723206, -0.3307029902935028, -0.17176856100559235, -0.2792856991291046, 0.7712673544883728, -0.30351194739341736, -0.2637748122215271, 0.1707724779844284], [-0.4406869411468506, 0.23599247634410858, 1.3678385019302368, -0.7524657249450684, 0.3077591061592102, 0.3051607608795166, 1.8917607069015503, -1.620894432067871, -1.0028703212738037, 1.1694952249526978, -1.4902622699737549, 0.3114748001098633, 2.2626562118530273, -0.19770729541778564, -0.46893855929374695, 0.9829847812652588, 0.45897480845451355], [-0.38027042150497437, 1.089609146118164, -0.583017110824585, -0.5124549269676208, -1.1493054628372192, -0.49415653944015503, 0.8172151446342468, 0.5340278148651123, 0.6129184365272522, 0.6351410150527954, -0.30939173698425293, 0.08938056230545044, -0.28756922483444214, 0.25706571340560913, -0.424078106880188, 0.7509042024612427, 2.3085672855377197], [-0.37799903750419617, -1.4796617031097412, -0.8043709397315979, -0.5658047795295715, 0.1653926521539688, 0.06704503297805786, 0.9977915287017822, 0.5126562118530273, -1.8421393632888794, 1.1959290504455566, 1.3261679410934448, -1.0688869953155518, -1.3038378953933716, -1.2384209632873535, -0.4121195077896118, -0.03523802012205124, -0.4977499842643738], [-0.38355425000190735, 0.837792158126831, -0.08912643045186996, -1.9503933191299438, 1.0310572385787964, 0.4804861545562744, -0.19619840383529663, -0.7291361689567566, 1.1387591361999512, 0.18395449221134186, -0.5774953365325928, 0.2858191430568695, -0.4938414394855499, -1.0857888460159302, -0.886925220489502, 2.0905284881591797, -0.5401394367218018], [0.9905725717544556, -1.4038972854614258, -0.41558709740638733, -0.7164372801780701, 0.6736325621604919, 1.3532559871673584, -2.083777666091919, -0.9962930679321289, -0.013036432676017284, -1.3397116661071777, -0.539482593536377, 0.36623111367225647, 0.5817595720291138, -0.4026663899421692, 1.3522316217422485, -1.8614526987075806, 1.186442494392395], [0.3053158223628998, 1.3071177005767822, -0.5874611139297485, -1.4294050931930542, -0.24236223101615906, 1.3581196069717407, 0.3090023696422577, -0.7380232810974121, -0.504805862903595, -2.0606777667999268, -1.7833914756774902, -2.188308000564575, 0.21571135520935059, -0.6989265084266663, -1.0138429403305054, -2.2413482666015625, 0.27012380957603455], [0.11766774952411652, -0.8583155870437622, 0.8161697387695312, -2.249901533126831, 0.1789345145225525, -0.4312989115715027, -0.39409536123275757, -1.7033755779266357, -0.995851993560791, -0.4926137328147888, 0.6584143042564392, -0.09015265852212906, 0.27878937125205994, -1.697975754737854, -0.6581165790557861, -0.3326463997364044, -0.5727275609970093], [-0.034219130873680115, -0.17321977019309998, 0.4064652621746063, -1.1335874795913696, -1.5920201539993286, -0.42590510845184326, 0.15700621902942657, -1.6240463256835938, -1.0213043689727783, 0.1462254375219345, 1.6809991598129272, -2.00226092338562, -0.7008123397827148, 0.9903299808502197, 0.5769428610801697, 0.6316540837287903, 0.4393157660961151], [-0.08984087407588959, -1.3366397619247437, 1.1527034044265747, 0.5351113080978394, -0.6451284885406494, 0.7793183922767639, 1.3644354343414307, -1.524005651473999, 1.171722650527954, 2.4654040336608887, -0.27193862199783325, 0.2468564212322235, -0.08292623609304428, -0.38459154963493347, 1.1755154132843018, -1.3850421905517578, -1.276784896850586], [-0.823852002620697, 0.3883381485939026, -1.0343495607376099, 0.870778501033783, 0.23267625272274017, 0.03962045907974243, -0.29585257172584534, 0.9571099281311035, -0.47934818267822266, -1.3979809284210205, -0.6366886496543884, -0.7317922711372375, 0.138315811753273, 1.0913282632827759, 2.2234811782836914, -0.9641523957252502, 0.5786094069480896], [0.6506471633911133, 1.378987193107605, 1.1125718355178833, 1.8028160333633423, -0.9974151849746704, 0.12237042188644409, -0.18972399830818176, -1.161934494972229, 0.09415420889854431, 2.0834577083587646, -1.4910507202148438, -0.6641504764556885, 1.1655969619750977, -0.31412675976753235, -0.5071842074394226, 0.8579034209251404, -0.730202853679657], [0.8597881197929382, 0.602549135684967, 0.9382185935974121, -0.6310120820999146, 0.36477190256118774, -0.1661626100540161, -1.3311586380004883, -0.20533610880374908, 0.69635009765625, 0.44483527541160583, -0.6727955341339111, 1.2914221286773682, -0.42473000288009644, 0.6938834190368652, -1.5098916292190552, 0.6030699610710144, 0.4782966375350952], [-0.20941393077373505, -0.6228043437004089, 0.8579111099243164, 1.2645262479782104, -0.23466956615447998, -1.9193806648254395, 0.4391914904117584, -0.9940990209579468, -0.6272994875907898, 0.4562634527683258, -0.32304203510284424, 0.3289327621459961, -0.029953310266137123, -0.07229994982481003, 0.035416606813669205, 0.10606516152620316, 0.14601008594036102], [-0.6718080639839172, -1.1870229244232178, 1.3699160814285278, 0.48529675602912903, 0.9254994988441467, -1.90998113155365, -0.39403772354125977, -1.2539564371109009, -1.6447429656982422, 1.7233765125274658, -1.1235284805297852, -0.550638735294342, -1.0914472341537476, 0.9855616092681885, 0.03399534896016121, -0.5516812801361084, -1.9083127975463867], [1.5435423851013184, 2.425309419631958, 0.2298615574836731, 1.3676589727401733, 0.44587430357933044, 0.06785420328378677, -1.2354086637496948, -1.1130386590957642, -0.9635173678398132, 0.017458736896514893, -0.34695181250572205, -0.5002143979072571, -1.0669742822647095, 0.4833447337150574, -1.6110002994537354, 0.9223599433898926, 0.4241582453250885], [0.5937486886978149, -0.8128553032875061, 1.1948212385177612, 0.7757848501205444, 0.8598254919052124, -0.962814211845398, 1.9780197143554688, 0.9636276364326477, -0.8584725856781006, 1.496290683746338, -1.872843623161316, -0.40476614236831665, 0.6340972781181335, 0.758635938167572, -0.29820582270622253, -0.4219949245452881, -0.14744779467582703], [-0.6583518385887146, 0.4057421386241913, -0.9696515202522278, 0.19097614288330078, -0.930802047252655, 0.4939388930797577, -0.7319718599319458, 3.143437623977661, -0.872225284576416, -0.3474876880645752, -0.3974747657775879, 0.03756559640169144, -0.41220760345458984, -0.45950183272361755, 1.2985788583755493, -0.36019131541252136, 0.675166666507721], [-1.238820195198059, -0.46229317784309387, 0.32181134819984436, -1.9154632091522217, 0.0769551619887352, 1.3408946990966797, -0.2523254156112671, -2.1211636066436768, -2.179809093475342, -0.03270649164915085, -0.6846466660499573, 0.4253365695476532, -1.206174373626709, -0.45610201358795166, 1.069338083267212, 0.18958978354930878, -1.6901215314865112], [-1.8235636949539185, -0.5008782744407654, 1.422101378440857, 0.017169199883937836, -1.9852962493896484, -0.5264573097229004, -0.9397221207618713, -2.76212477684021, -0.19383323192596436, 1.331977367401123, 1.2409446239471436, -0.8177214860916138, 0.06632496416568756, 0.3073010742664337, 0.6711320281028748, 1.2307549715042114, -2.6360127925872803], [-0.5647881031036377, -2.0494513511657715, -1.3775562047958374, 0.8876864314079285, -0.8116671442985535, -2.240170478820801, -1.0838218927383423, -0.7976226210594177, -1.3071266412734985, -0.07707834243774414, 0.658833384513855, 1.1235772371292114, 0.006289804819971323, 0.17041651904582977, 2.352304220199585, -0.22740498185157776, 0.10805222392082214], [-0.8377472162246704, 1.358921766281128, 0.8498886823654175, 0.7574026584625244, -0.3172999620437622, 0.48262444138526917, 0.510138750076294, -1.1890149116516113, 0.5541314482688904, -0.41513556241989136, 0.965259313583374, 1.6469370126724243, -0.5249804258346558, 0.5379753708839417, 0.7952936887741089, 0.07591541111469269, 0.4839079678058624], [-1.397541880607605, -0.3477191925048828, -0.9107194542884827, -0.056118324398994446, 0.3614377975463867, 0.394095778465271, -1.2926279306411743, 0.41741886734962463, -0.6429517865180969, 0.16800978779792786, -0.2694057524204254, 0.7109938859939575, 0.5771994590759277, 0.13799160718917847, -0.2981780767440796, 0.9443221092224121, 1.3084884881973267], [-0.26917609572410583, -1.4149147272109985, -0.026693567633628845, 0.5862656235694885, -0.7366791367530823, 0.8450320959091187, -0.6558753252029419, -0.37100255489349365, 0.44967660307884216, -0.11512168496847153, 0.2491215318441391, 1.6654685735702515, 0.7777734994888306, -1.5892494916915894, 0.042714186012744904, 0.954992949962616, 0.39024242758750916], [-1.1660236120224, 1.3358334302902222, 1.4573079347610474, 0.7693908214569092, -1.2039408683776855, 0.07143665850162506, -0.05989072844386101, -0.3640032708644867, 0.7238113880157471, -0.6724917888641357, 0.005460833199322224, 0.09545999020338058, 0.5439367890357971, -1.078606128692627, -1.304433822631836, -1.652569055557251, 0.5136218667030334], [0.8490573167800903, -1.2765085697174072, -0.018057703971862793, 0.6104615926742554, -0.43004605174064636, -1.348059892654419, 0.3249081075191498, -0.4886561930179596, -0.5167196989059448, -0.3652067184448242, -1.4236968755722046, -0.9379399418830872, 0.8301336169242859, 0.45593270659446716, 2.6338565349578857, 0.5883021354675293, -0.10435017198324203], [-0.5641825795173645, -1.7471880912780762, -0.05340147390961647, 0.3993682563304901, 1.2550994157791138, -0.23911704123020172, 0.8593056201934814, 2.653779983520508, 0.13289617002010345, -1.2643269300460815, 0.3727650046348572, 0.044232308864593506, 2.3175442218780518, 0.541051983833313, -0.7420478463172913, -0.24043181538581848, -1.5498781204223633], [0.2469703108072281, 1.356589913368225, 0.649275004863739, 0.3874717056751251, 0.4444824159145355, 0.1839924156665802, 0.42006900906562805, 2.5215940475463867, -0.5844236612319946, -0.7414712905883789, 1.0255606174468994, 0.11709374189376831, 0.7214664816856384, -0.1629142016172409, 0.7668248414993286, 0.7470959424972534, -1.869214653968811], [-1.0616364479064941, -0.07988444715738297, -0.5791921615600586, 0.036393601447343826, 0.31076496839523315, 1.785742998123169, -0.6992933750152588, 0.385591596364975, -0.3695737421512604, -0.9397532939910889, 0.6712806224822998, 0.2927163243293762, 0.19587381184101105, -0.29094359278678894, 1.7555421590805054, -1.1394577026367188, 0.5725470781326294], [-0.17772029340267181, -0.315519243478775, 0.07396458089351654, -0.4335690140724182, 0.41620567440986633, 1.510704517364502, 0.7767836451530457, -1.1030021905899048, -0.5004909634590149, -0.5664607286453247, -0.5264395475387573, 1.1818768978118896, 1.637373447418213, 0.35210877656936646, 1.865546464920044, -2.512653112411499, -0.045314185321331024], [-1.3031456470489502, -0.622435986995697, -0.9615933895111084, -0.2312086969614029, -0.38853946328163147, -1.8886350393295288, 1.343485713005066, 0.48724111914634705, -0.10045695304870605, 0.39497336745262146, 0.36531198024749756, 0.04185562580823898, 2.1031947135925293, 1.665013313293457, -0.15373434126377106, 0.3986966907978058, 0.3936910927295685], [-0.610521674156189, -0.17991049587726593, 0.44637855887413025, -1.0066642761230469, -0.555270791053772, 0.1696692705154419, 0.2855086922645569, -1.438340663909912, 1.5624512434005737, 0.34443825483322144, -0.8939852118492126, -1.1346527338027954, -1.383096694946289, -0.18486826121807098, -0.6119332909584045, -1.767960548400879, -0.417862206697464], [-1.330132007598877, 0.2545566260814667, -0.0038556696381419897, -0.6340786814689636, -0.07751035690307617, 0.30007484555244446, -0.019033806398510933, 0.4145548343658447, -0.5446186065673828, -0.12248940765857697, -0.21539638936519623, -0.16973598301410675, 1.3063852787017822, 0.18615370988845825, 1.9404933452606201, 0.5752955675125122, -0.6344950199127197], [-0.2931806147098541, -0.7938481569290161, -1.1547486782073975, -0.06012523174285889, -1.4500625133514404, -0.2169829159975052, -0.38307276368141174, 0.5281258225440979, 1.0331556797027588, -0.35157665610313416, -2.1114180088043213, -1.4763984680175781, 1.7364460229873657, 0.16257955133914948, -2.130431890487671, -0.7104815244674683, 0.36649930477142334], [-0.018135448917746544, -0.8501730561256409, 1.1609125137329102, -0.6935927867889404, 0.9281246066093445, -0.37518003582954407, 0.29524487257003784, 0.3996283710002899, -1.2731033563613892, 0.6797686815261841, 1.170236349105835, 1.4077811241149902, -1.1335816383361816, -0.6864309906959534, 1.087389588356018, 1.9402250051498413, -0.16260521113872528], [1.447013020515442, -0.07431622594594955, -0.45549729466438293, -0.6149686574935913, 0.5424113273620605, -1.1594589948654175, -1.7966867685317993, 0.22356198728084564, 0.6097877025604248, 0.9570751190185547, 0.900297224521637, -0.7404277324676514, -0.3340550363063812, 1.7080096006393433, 0.9340229034423828, 0.8892319798469543, -0.8010485172271729], [2.502487897872925, 1.4013251066207886, -0.32401296496391296, 1.4882574081420898, -0.25735145807266235, 1.322989821434021, -2.504445791244507, 0.7671990394592285, -0.04479704424738884, 0.8208102583885193, 0.25894731283187866, -0.7234367728233337, 1.01679265499115, 1.0228205919265747, -0.9144350290298462, 0.4593171775341034, 0.2267989069223404], [-0.18249095976352692, -0.13538885116577148, 0.6677977442741394, 0.6640194058418274, 0.45120689272880554, 1.510136604309082, 0.40225690603256226, -0.867210865020752, -0.50542151927948, 0.24343015253543854, 2.2849843502044678, 0.10378868877887726, -1.263559103012085, 0.22682012617588043, 1.3241606950759888, 0.9447504878044128, 0.6539344191551208], [0.5268317461013794, 0.25778958201408386, 0.9065561890602112, -0.9211585521697998, 0.5755612850189209, -0.20141418278217316, -0.8069037795066833, 0.03278706595301628, -0.9918937683105469, -0.9229167103767395, 0.43854936957359314, -1.8847736120224, 1.3707391023635864, 0.12971296906471252, -0.4825868606567383, -1.4026259183883667, 0.510816216468811], [0.1928364336490631, -0.11730558425188065, -0.3265179693698883, 0.6021316647529602, -0.9037428498268127, 0.31634965538978577, -0.3721628785133362, -1.7799668312072754, -0.2698706388473511, 0.7769056558609009, 0.40355169773101807, -1.5950802564620972, 0.2678848206996918, 0.8250998854637146, 0.9281664490699768, 0.10659745335578918, 0.6626824736595154], [-0.4564220607280731, 1.129905343055725, -0.3175511062145233, -0.7839012742042542, -1.0345646142959595, 1.0857023000717163, 0.9014340043067932, -0.6923729181289673, -0.351375013589859, 0.744499921798706, 0.2568019926548004, 0.4201301336288452, 0.472708523273468, 0.1908525824546814, -0.8133045434951782, -0.5494348406791687, -1.5866806507110596], [-0.15966689586639404, 0.16166740655899048, -0.77414470911026, 0.542446494102478, 0.4317297041416168, 0.6941222548484802, -1.0922737121582031, -0.8791720271110535, -0.3883083462715149, -0.8244088292121887, -0.5591343641281128, -0.8113395571708679, -1.389190912246704, 0.7739925384521484, 0.12125901877880096, 0.45477816462516785, 0.7448174953460693], [0.0618186891078949, 1.1445674896240234, 1.6173819303512573, 0.9542726874351501, 1.8716397285461426, 0.08561952412128448, -0.8585656881332397, 0.38498756289482117, 0.34606099128723145, -0.8539057970046997, 0.14938336610794067, 0.5089761018753052, 0.3835591673851013, -0.849376380443573, -2.1058390140533447, 0.496272474527359, -0.7965108752250671], [-0.5588800311088562, -0.1990455538034439, -0.17188884317874908, -0.10494701564311981, 0.5335813760757446, 1.3428510427474976, -1.2484638690948486, 1.5558838844299316, -1.2716121673583984, -0.04601132124662399, 0.6527440547943115, -1.6281507015228271, 1.2663590908050537, 0.44103699922561646, 0.4951207637786865, 0.438414067029953, 1.3793821334838867], [1.1385866403579712, -0.4723716974258423, -0.49262455105781555, 0.20032523572444916, -0.3873467445373535, -0.4118789732456207, 0.9538979530334473, -1.106888771057129, -0.04575274512171745, 1.0903443098068237, 0.20383189618587494, -0.7120605111122131, -1.0642881393432617, 0.5808207988739014, -0.5803364515304565, 0.4197060167789459, -0.9690393805503845], [-0.05164994299411774, 1.7942390441894531, -2.7123334407806396, -1.3875083923339844, -0.3332905173301697, -0.4434836208820343, -0.15480637550354004, 0.0612412691116333, 1.2868413925170898, 0.8920284509658813, 0.9873946309089661, -1.4198299646377563, -0.5104170441627502, 1.5874741077423096, 1.1375809907913208, -0.609015166759491, -0.4969208836555481], [1.0240709781646729, 0.37558746337890625, 1.5274696350097656, -0.7710110545158386, 0.8908358216285706, 0.10762876272201538, -1.2144813537597656, 0.08767557144165039, 1.0498688220977783, -0.6771329641342163, 0.93248450756073, -0.4995090663433075, -0.9547910094261169, -0.7464767098426819, 1.1844556331634521, 0.03305840119719505, -1.338287353515625], [-0.02782050147652626, -1.5283600091934204, -1.1986585855484009, -0.38507503271102905, -0.8183599710464478, 0.5113195776939392, 1.12714421749115, 1.384060025215149, -0.2987782955169678, -1.3349761962890625, -0.7355998158454895, 0.797808051109314, 0.5823985934257507, 1.1485284566879272, 0.42436814308166504, 0.6198675036430359, 0.9957599639892578], [-0.419373095035553, 1.1289299726486206, -1.4654393196105957, 0.6059096455574036, -1.0587152242660522, 2.370100975036621, 0.26060548424720764, -0.7927361726760864, 1.3924710750579834, -1.552027702331543, -0.6135934591293335, -1.6904457807540894, 1.2488462924957275, -1.12246572971344, 0.9993568062782288, -0.4369927644729614, 0.21579304337501526], [-0.9072871804237366, 1.0128201246261597, 1.670230507850647, -0.6097282767295837, 0.4380209445953369, -1.6016627550125122, 1.909481406211853, 0.22411099076271057, -1.1389806270599365, 0.12172351777553558, -0.8579647541046143, -1.9554654359817505, -0.5568135976791382, -0.5497758388519287, 1.2630019187927246, 0.6852447986602783, -0.5631288886070251], [-0.2717325687408447, -0.5559722781181335, 0.6658890247344971, -1.1231476068496704, -0.8228709101676941, -0.6493252515792847, -0.47522661089897156, 0.1744689643383026, 0.060888588428497314, 0.19241850078105927, -0.7535053491592407, -0.876623809337616, 1.9378483295440674, 0.23545081913471222, 0.08723651617765427, 0.36864590644836426, -1.2711148262023926], [-0.36969465017318726, 0.18752066791057587, 0.7026504278182983, -0.4113388955593109, 0.9287747144699097, -0.18860109150409698, -0.30206528306007385, -0.02534325420856476, 0.11895626038312912, -0.17590384185314178, 1.2934414148330688, -0.3156301975250244, -0.5128616690635681, -0.13111357390880585, -0.29968658089637756, -2.234391212463379, -0.1759594827890396], [-0.07848978042602539, 0.7812948822975159, -0.5687009692192078, 2.588193416595459, -0.03469850495457649, 1.5617828369140625, 0.7549231052398682, 2.218996047973633, 1.0357328653335571, 0.2656484544277191, -0.5191822648048401, 0.3967040777206421, -0.34325680136680603, 2.0392627716064453, -0.2243485152721405, -0.9149877429008484, -0.6700587272644043], [0.03744659945368767, 0.951690137386322, 0.6362335085868835, 0.46920737624168396, 0.07955356687307358, -2.0159356594085693, 1.4121196269989014, 0.058874279260635376, 0.5368891954421997, 0.2694849371910095, -1.4880797863006592, -1.1966356039047241, 0.20360130071640015, 0.3847137689590454, 0.8241634368896484, 0.5196534991264343, 1.968308687210083], [0.600385308265686, 0.200557678937912, 1.817631483078003, 0.5224481821060181, -0.30779311060905457, -1.1945815086364746, -0.2449248731136322, 1.0632246732711792, 0.4268498420715332, -0.41162121295928955, -0.9369027614593506, 1.098430871963501, -1.2970482110977173, 0.8556699752807617, -0.051007866859436035, 0.3206689953804016, 2.4719810485839844], [-0.6531062722206116, -2.7416419982910156, -0.5182452201843262, 1.4555110931396484, 0.09211931377649307, -0.33907994627952576, 0.09817708283662796, 1.1431808471679688, -0.39062270522117615, 0.9663163423538208, -0.5820121169090271, -0.693332850933075, 0.4055251181125641, -1.4336743354797363, 0.012101700529456139, 0.05471659451723099, 0.607633650302887], [-1.5931261777877808, 2.392998695373535, -0.848829984664917, 0.6421104073524475, 0.32417237758636475, -0.4333604574203491, 0.04607312008738518, 0.15622839331626892, -0.3840232491493225, 0.6773263216018677, -0.149630606174469, 1.0906777381896973, -1.031897783279419, 1.1236995458602905, -0.2602955996990204, -0.006274379324167967, -0.602433443069458], [0.7202631235122681, -0.14626780152320862, 0.10732270032167435, 0.27063503861427307, 0.22667130827903748, 0.13492192327976227, -2.154296398162842, 0.19798481464385986, 0.5567796230316162, 0.23202753067016602, -0.35642698407173157, 0.5228789448738098, 0.8693298101425171, 0.2171436995267868, 0.8885183930397034, -1.9469871520996094, -0.6961541771888733], [0.018049102276563644, -1.2455836534500122, 0.1323552429676056, -1.5601240396499634, -0.5241952538490295, -0.22816066443920135, 1.1607462167739868, 1.1955773830413818, -0.4091952443122864, -1.1099976301193237, 0.29612553119659424, -3.1627390384674072, -0.9998590350151062, 0.8749546408653259, 0.6529260873794556, 0.9644368290901184, 0.5556395649909973], [0.6679401397705078, -0.015804056078195572, 0.061172280460596085, 0.2612237334251404, -0.7854843139648438, 1.4726037979125977, -0.22742363810539246, 0.28724268078804016, -0.8050078749656677, -0.055127017199993134, -0.08129098266363144, 2.0576999187469482, 2.417383909225464, -2.511859655380249, -2.7547249794006348, -0.055958040058612823, 0.005523446947336197], [1.1491649150848389, 0.5158064961433411, -0.16128645837306976, 0.04156059771776199, -0.4759867787361145, -0.2817276418209076, 2.1504456996917725, -0.703752875328064, 2.4503073692321777, 1.459096074104309, -0.7038283944129944, 0.2940850853919983, -0.3750775456428528, 1.235770583152771, -1.2737618684768677, -0.5154695510864258, -0.3081279993057251], [-0.32998040318489075, 1.2003629207611084, 0.19150249660015106, -0.5894288420677185, 0.7798216938972473, 0.4906689524650574, -1.7929315567016602, -0.4967615306377411, -0.059750035405159, -0.0894717276096344, -1.0922386646270752, 0.1921546906232834, -0.5706756711006165, 1.014109492301941, -0.8601030707359314, 0.1294969618320465, -0.4392690360546112], [0.4502143859863281, -1.1422544717788696, -0.9300649762153625, -0.24755361676216125, 0.4490390419960022, -0.42171505093574524, -1.258448839187622, 0.21704818308353424, -0.4245047867298126, -0.7545763254165649, -0.10999386012554169, -0.4308494031429291, -0.9397898316383362, 0.10816098004579544, 0.5959527492523193, 1.2066336870193481, 0.8583030700683594], [-0.04022600129246712, 0.2830718755722046, 0.2402738481760025, -1.894771695137024, 1.9547052383422852, -0.7476523518562317, 0.6509740948677063, -0.6888492703437805, -0.855556845664978, -1.3954930305480957, -0.8193788528442383, 0.8235475420951843, 0.5120834112167358, -0.1860426515340805, 0.4730798602104187, 1.6711227893829346, -0.25551024079322815], [-0.17520497739315033, -0.5677101612091064, 0.24130891263484955, 0.9018965363502502, -0.46558868885040283, 0.2051129788160324, -0.5242788791656494, 0.4259299039840698, -0.46024787425994873, -0.7006034851074219, -0.6427739858627319, -1.6813733577728271, -1.7621709108352661, -0.014849888160824776, -1.3711668252944946, -1.8439770936965942, -0.5035948753356934], [0.3744300603866577, 0.14363601803779602, -1.0112608671188354, -0.40350085496902466, 0.8795459270477295, -0.8818274736404419, -1.0441590547561646, -0.9099375605583191, 0.7857683897018433, 1.0210047960281372, -0.9384597539901733, 0.653235673904419, -0.7456365823745728, -0.8586782813072205, 0.7852128744125366, 1.157794713973999, -0.45650994777679443], [0.9249587059020996, -0.8519773483276367, 0.2904965281486511, -0.7270467281341553, -1.257033348083496, -0.1984027475118637, 0.6959523558616638, 0.5719009041786194, 0.28092530369758606, 1.3915742635726929, -0.494897723197937, 0.35946744680404663, 1.9284729957580566, 1.2269020080566406, 0.424079030752182, -0.6002314686775208, 0.2761732339859009], [0.8014779090881348, 0.2796327471733093, 0.1754651665687561, 0.3294464647769928, -1.781636357307434, 0.827988862991333, 0.044752348214387894, -0.12270069122314453, 0.6123677492141724, 0.9330909252166748, 0.3126133382320404, 0.1890217810869217, -1.2470369338989258, 1.3652158975601196, -2.076679229736328, -0.8686305284500122, 0.01993151195347309], [0.10668739676475525, 0.15824642777442932, -0.6131576895713806, 0.9667364358901978, -0.8307060599327087, -0.6252274513244629, 0.043328963220119476, -0.5824598670005798, -1.8280067443847656, -1.2208812236785889, -1.3433305025100708, 1.3848307132720947, 0.01298774778842926, 0.023565959185361862, -0.4688204824924469, -0.2819393575191498, 1.5393146276474], [-1.1321446895599365, -0.8774726390838623, -1.4324051141738892, -1.5153695344924927, -1.2770755290985107, -0.040454257279634476, -0.009733205661177635, -1.3038361072540283, -0.5105789303779602, -1.5393626689910889, -1.9262824058532715, -0.648853600025177, 0.6543879508972168, -1.1405024528503418, 0.1156921535730362, -1.459000825881958, 0.15108659863471985], [-0.21267962455749512, 0.5503293871879578, 0.7146633863449097, 1.6089407205581665, -1.4672847986221313, 0.8337016701698303, 0.06792223453521729, -0.4808947741985321, -1.8442648649215698, 0.9936686754226685, 1.2970303297042847, -1.4758760929107666, -0.4669985771179199, 1.4474372863769531, 1.2844252586364746, 1.1067270040512085, -0.9372516870498657], [1.5320748090744019, 0.7048055529594421, -0.2945912480354309, 0.5799767374992371, -0.4439921975135803, -0.5100780129432678, -0.8277637958526611, 0.3329695165157318, 0.8972826600074768, 0.6842861771583557, 0.021290447562932968, -2.4226717948913574, 0.03599518537521362, 1.6910227537155151, 0.5109890699386597, 1.6269172430038452, -1.0471501350402832], [-1.0361998081207275, 0.1731034368276596, -0.1584002524614334, 0.934230625629425, 0.3669934570789337, 0.2552178204059601, -1.0005038976669312, 0.0070617529563605785, 1.100289225578308, -0.18678735196590424, 0.12743985652923584, 0.4560171663761139, -0.23287172615528107, 0.4551675021648407, 1.527601718902588, -0.6358466148376465, -0.575632631778717], [-1.6527862548828125, -1.0171473026275635, -0.1475585550069809, -0.34929296374320984, 0.37894490361213684, 0.3511207401752472, 0.23313386738300323, 0.03319976478815079, -0.29078173637390137, -0.11555525660514832, -1.3752131462097168, 0.8153544068336487, -1.1083160638809204, 0.07440373301506042, 1.1465729475021362, -0.09083830565214157, 1.3317182064056396], [-0.8819114565849304, 0.5749983787536621, -0.7208978533744812, 0.6766231060028076, 0.3297661542892456, -0.5339635610580444, -0.0064651137217879295, -0.6556400060653687, -0.9250739216804504, 0.0030801037792116404, 0.35327330231666565, -0.6259578466415405, -1.449845314025879, -0.6271238327026367, -0.46413999795913696, -1.0870424509048462, 0.6835959553718567], [-0.7911331653594971, 1.1995981931686401, -0.03298414126038551, 0.2112870067358017, -0.11118989437818527, -1.6245731115341187, -0.7058705687522888, -0.43051251769065857, -0.5540581941604614, 1.854523777961731, 0.7257406115531921, -0.6053727865219116, -1.9507920742034912, 0.3969348669052124, -1.6765931844711304, -0.05589238181710243, 1.6113942861557007], [2.902885675430298, 2.35449481010437, -0.5405980944633484, 1.4027693271636963, -0.11643482744693756, 0.011034431867301464, 0.5032488107681274, -0.15445780754089355, -0.1872912049293518, 0.1572418212890625, -2.543916702270508, 0.40464887022972107, 0.6323156952857971, -0.23334555327892303, 0.37517601251602173, 1.4573540687561035, 0.3791975975036621], [0.14449360966682434, 0.176479771733284, 1.8203673362731934, 1.378684163093567, -0.5181010961532593, 1.7276718616485596, 1.3610726594924927, 0.9077722430229187, -2.1219019889831543, -0.07973065227270126, 0.6325049996376038, -1.4329134225845337, 0.7342596650123596, 0.9331682920455933, -0.7683404684066772, -1.334301233291626, 0.4030623137950897], [-1.5434211492538452, -0.4671262800693512, -0.9064249992370605, -1.063085913658142, -0.09737623482942581, 0.26718416810035706, 0.35856205224990845, 1.4266525506973267, 0.9613252878189087, -0.7088147401809692, 0.3434002697467804, -0.5877964496612549, -1.6926841735839844, 1.1588940620422363, 0.4242575466632843, -0.4063640534877777, 0.810125470161438], [0.6674320101737976, 0.397623747587204, -0.2445179522037506, -0.9561095833778381, 0.4686443507671356, -0.18940506875514984, 0.9463571310043335, -0.12762203812599182, 0.9237588047981262, -0.6014370918273926, -0.36410075426101685, -0.6178877353668213, 1.19270658493042, -0.17739294469356537, 0.16371802985668182, -0.4291532635688782, -1.038222074508667], [-1.496665358543396, -0.9682649374008179, -0.13215431571006775, 2.3168296813964844, 1.2728393077850342, -0.6901196241378784, 1.1159977912902832, 0.0027939036954194307, 1.3472051620483398, 0.201368510723114, -0.7944653630256653, 0.1775364875793457, 0.32856398820877075, 0.391628235578537, -1.5042275190353394, -0.3885253667831421, 0.8818996548652649], [0.5824342966079712, -1.0178892612457275, -0.47921574115753174, 0.6250155568122864, 1.2376683950424194, 0.41396069526672363, -1.047003984451294, 0.5538994669914246, 1.2405683994293213, -0.45507746934890747, 0.3232465386390686, 0.700660765171051, 0.025289667770266533, -0.3682449460029602, 1.19049870967865, -0.02253597043454647, -0.8302983641624451], [-0.14981591701507568, -1.251338005065918, 1.0239574909210205, 0.19439634680747986, 0.20695187151432037, -0.2626384496688843, 0.6819925904273987, -0.7102178335189819, -0.026688262820243835, -1.3913356065750122, -0.9103997349739075, -0.9040693640708923, -0.6968701481819153, -1.4269641637802124, 1.7398109436035156, 0.38628751039505005, 0.4024435877799988], [-0.9030110239982605, 0.1693795621395111, 2.0138261318206787, -0.5578886270523071, -0.18961448967456818, -0.5648150444030762, -0.8675299882888794, 0.30538567900657654, 0.4871392250061035, -0.633229672908783, 0.9547330141067505, -0.3931407034397125, 0.17964287102222443, -0.7930664420127869, 1.415145993232727, -0.11175987869501114, 0.07298875600099564], [0.45764830708503723, 0.7921339273452759, 0.6319042444229126, -0.40945059061050415, -0.44222432374954224, -0.11107177287340164, 0.2845856845378876, -0.026566561311483383, 3.8955507278442383, 1.3725513219833374, 0.04671727493405342, 0.028556732460856438, 0.407713919878006, -0.7080246210098267, 1.6077641248703003, 0.7909432649612427, 0.15255218744277954], [0.3623199164867401, 0.17240716516971588, 0.45717722177505493, 1.5566407442092896, 2.3041045665740967, 0.5975310802459717, 2.1917226314544678, -0.498868465423584, 1.1726181507110596, -1.0734741687774658, 1.1822280883789062, 0.1877710074186325, -1.7067875862121582, 0.5014707446098328, 1.4640110731124878, 0.25302770733833313, 0.8330255746841431], [-1.2943952083587646, -0.8285874724388123, 0.5924398303031921, 0.21992436051368713, 0.672683835029602, -1.1997956037521362, 0.6165351271629333, -0.916454017162323, 0.32587185502052307, -0.1714717298746109, -1.1183562278747559, 1.0230193138122559, 0.7031288146972656, 0.6030328869819641, -0.9125986695289612, -0.04961583390831947, -1.0026299953460693], [-0.11671812832355499, -1.2854194641113281, -0.5801248550415039, -0.9252866506576538, -0.7142530679702759, 0.004291156772524118, 1.9695076942443848, 1.4306840896606445, -0.7727358341217041, -0.3619861304759979, -0.9853596687316895, -0.6997760534286499, -1.4661462306976318, 0.1917700618505478, 1.8554998636245728, 0.231976717710495, 0.35948583483695984], [0.5993906259536743, 1.0086190700531006, -2.07033109664917, 1.0613102912902832, 0.6515243053436279, -0.44534268975257874, 1.132431983947754, 1.327254295349121, 0.9192846417427063, 0.1793711632490158, -0.7082278728485107, 0.29842039942741394, 0.7286103367805481, -0.43317869305610657, 0.9410390257835388, 0.5318771004676819, -1.0339624881744385], [-0.7732992172241211, 1.0235099792480469, -0.9738085269927979, 1.6082098484039307, -0.06504744291305542, -0.5190611481666565, 0.052148301154375076, -1.2764390707015991, 0.19190745055675507, 1.0751347541809082, -1.0789934396743774, -1.0479731559753418, 0.6170029640197754, -0.12787720561027527, -0.25916555523872375, 1.3833318948745728, -0.1237461194396019], [0.6429527401924133, 1.1022268533706665, -0.08925364166498184, -0.24854853749275208, -1.2965871095657349, -0.29008880257606506, -0.4974590241909027, 2.4775023460388184, -0.4495376944541931, 1.9523087739944458, -0.00987065676599741, 0.2621322572231293, 0.8038291335105896, -0.378274142742157, 0.8780755996704102, 0.2857756018638611, -1.0448989868164062], [0.12396953254938126, 1.9231610298156738, 1.1393539905548096, -1.5987606048583984, 1.906902551651001, -0.8487966656684875, -1.7243009805679321, -0.7059358358383179, -0.5790653824806213, -0.8809154629707336, 1.3165967464447021, -1.6229841709136963, -0.5113451480865479, -0.5084952712059021, -0.32675817608833313, 0.8992097973823547, 1.3265445232391357], [1.7173882722854614, -0.3796718716621399, 0.7570730447769165, 1.761947512626648, 0.7356508374214172, -0.042842548340559006, -0.27706217765808105, -1.0685368776321411, -1.116714358329773, -1.600548505783081, -1.0573550462722778, -0.29440760612487793, -0.5295836925506592, -0.809488832950592, 0.1998683214187622, 0.9894039034843445, -0.6136777400970459], [1.4305006265640259, -0.5892449021339417, -0.4428349733352661, 0.2331668734550476, -0.7011313438415527, 1.0871920585632324, -0.5206552147865295, -1.072019338607788, 0.04976104944944382, 0.9344977736473083, 0.1856384426355362, 0.09552855789661407, -1.551483154296875, 1.4839013814926147, 0.4801008701324463, -0.6492982506752014, -0.29082489013671875], [1.1329798698425293, 0.9657891392707825, -0.2039397954940796, -1.286791205406189, 1.560451626777649, -1.2256020307540894, -0.41493120789527893, -1.490073323249817, -0.7457790374755859, -0.5428428053855896, 1.1893824338912964, 0.6969255805015564, -0.7727814316749573, -0.6071207523345947, -0.5199976563453674, 0.7043101787567139, 0.43561863899230957], [0.13021598756313324, 0.5842848420143127, -0.3503073751926422, -2.7668304443359375, 0.13990752398967743, 0.786704421043396, 0.13535699248313904, -0.6284822821617126, 0.5472511053085327, -0.15082010626792908, -1.15076744556427, 0.012683754786849022, 0.9347525835037231, 0.37512385845184326, 0.6798251271247864, 0.6718703508377075, -1.2841154336929321], [1.1518282890319824, 0.294181227684021, -0.9650967717170715, 0.7620511651039124, 0.24989047646522522, -0.49208539724349976, -0.3350975811481476, 1.1986256837844849, 1.227316975593567, -1.7704347372055054, 1.1440550088882446, 2.2529032230377197, 0.43258851766586304, 0.17512661218643188, 0.38366857171058655, 2.7513082027435303, 1.8580635786056519], [0.6105238795280457, 1.1320923566818237, -0.8525902032852173, -1.3547191619873047, -0.28801366686820984, -0.8573475480079651, 0.0011451148893684149, -0.5294443368911743, 0.6648733615875244, 0.6793660521507263, -1.8934978246688843, 0.37067559361457825, 0.48542481660842896, 0.007301167119294405, 0.20721499621868134, 0.7791187763214111, -0.6968158483505249], [0.2833232581615448, 0.5060141086578369, 1.6039677858352661, -0.11736606806516647, 1.157416582107544, 0.5055298209190369, 0.10376697033643723, -0.6631418466567993, -1.7482198476791382, -0.9905759692192078, 0.002875030040740967, -1.9960912466049194, -1.6276737451553345, -0.8228446841239929, -0.07442309707403183, 0.6379274725914001, -0.9564383625984192], [-0.4424341320991516, 1.3807216882705688, -1.0592296123504639, -0.9052225947380066, 0.19342097640037537, 0.15148255228996277, 1.3631342649459839, 0.07735033333301544, -0.2413036972284317, -1.4654167890548706, 0.5920858979225159, 0.6489341855049133, 0.49991583824157715, -0.38791221380233765, -0.03547991067171097, 0.8579168319702148, -0.02906550094485283], [-1.4246100187301636, 0.9101881980895996, 0.018286535516381264, 2.01840877532959, 0.7751736640930176, 0.5664175152778625, -0.4780026376247406, -1.8447201251983643, -0.6815374493598938, 0.2777879238128662, 0.7769131064414978, -0.6891239285469055, 0.7979732751846313, 1.0321648120880127, 0.7507551908493042, 1.5719544887542725, -1.1626418828964233], [0.7345454692840576, 0.5175624489784241, -1.3464205265045166, 0.35867753624916077, 0.596448540687561, 0.3687993288040161, -0.11137819290161133, 1.488480806350708, -0.8745778203010559, -1.9700380563735962, 0.4690980911254883, 0.6931805610656738, 1.675718903541565, 2.1267292499542236, -0.34563148021698, 0.5103831887245178, -0.335130900144577], [-0.482208788394928, 0.25642117857933044, -0.24122415482997894, -0.14772120118141174, 0.46475112438201904, 1.4625437259674072, 0.6906770467758179, 0.7204375863075256, 0.5740232467651367, 0.0725337564945221, -0.06463892757892609, 0.3715806007385254, -1.9037234783172607, -1.791239857673645, -0.636449933052063, 1.9749863147735596, -0.7508822679519653], [1.5167019367218018, -0.9784436225891113, 0.833872377872467, 1.0259389877319336, 0.2882658839225769, 0.05401209741830826, 0.7086590528488159, -0.12872670590877533, -0.6445778012275696, 0.3751774728298187, -0.210200697183609, 0.9619999527931213, 0.4262617528438568, -0.5040549039840698, -0.9267998933792114, -0.9401947259902954, -0.9495188593864441], [-0.8709186911582947, -0.8567407131195068, 0.46258196234703064, -1.6821699142456055, -0.5464590787887573, 1.042668104171753, -0.6672270894050598, -0.6559045314788818, 0.8334746956825256, -0.10434597730636597, -0.5154688358306885, -0.23367883265018463, 0.23109136521816254, 1.2903200387954712, -0.3322505056858063, 0.6771476864814758, 0.25651147961616516], [1.1377328634262085, 1.0891999006271362, -0.28869494795799255, -0.45208078622817993, -0.1444777101278305, 0.0018334633205085993, 0.2455483376979828, -0.5530174374580383, 0.3764946758747101, 1.3202147483825684, -0.9590948224067688, -0.9173004031181335, 0.8715868592262268, 0.5400569438934326, 1.1760382652282715, 0.4533716142177582, 0.4099313020706177], [-1.433534860610962, -1.9180500507354736, 0.3468305468559265, 0.1321467012166977, -0.0035387659445405006, 0.9469687938690186, 0.5295175313949585, -2.122145414352417, -0.20469394326210022, 0.676289975643158, 1.7315919399261475, 0.8112860918045044, -0.2019123136997223, 1.251715898513794, 0.06714174896478653, -0.46740370988845825, -0.5295457243919373], [-0.0874553993344307, 0.83098965883255, -0.5941390991210938, -0.13334441184997559, 1.1955797672271729, -0.1815948486328125, -0.9760854244232178, -1.3011798858642578, 1.1161571741104126, 2.1854190826416016, 0.31659388542175293, -1.2893755435943604, 0.09285637736320496, -0.7090837955474854, -0.40934374928474426, -0.38495132327079773, -0.439980685710907], [-2.0738472938537598, 1.0154112577438354, 0.017140423879027367, -0.7028223872184753, 0.5075854063034058, 0.14777414500713348, -0.5742642879486084, -0.38908088207244873, 0.22407442331314087, -0.7808830142021179, 0.6313410997390747, 1.0399717092514038, -0.014518779702484608, -1.9127768278121948, -0.07972993701696396, -1.0027127265930176, 0.9502109885215759], [-0.2520451545715332, 0.7639828324317932, 0.04751163721084595, 0.6249985098838806, 0.594969630241394, -0.14658673107624054, 0.5935555100440979, 0.24134160578250885, -0.17228753864765167, 0.27966004610061646, -1.12729012966156, -0.3947097063064575, 2.0553131103515625, -0.9001955986022949, -1.386274814605713, 1.0947136878967285, -0.14273126423358917], [0.5029928684234619, -0.20201100409030914, 0.17252415418624878, 1.9178253412246704, -0.2992153465747833, 1.6933668851852417, 1.9673768281936646, -0.8759533762931824, -0.6830220222473145, -1.4527926445007324, -1.113138198852539, -0.814705491065979, 0.3988301753997803, 0.4229433834552765, 0.2466963231563568, 0.24619528651237488, 0.21350739896297455], [0.25137513875961304, -1.9399635791778564, 1.1184372901916504, -0.5186253190040588, 0.05994763225317001, 0.0037134571466594934, -0.9602822065353394, 0.47897541522979736, -0.6444557905197144, -1.3162287473678589, -0.8399782776832581, -1.118249773979187, 0.6534530520439148, 0.4373924434185028, 0.7368496656417847, -1.1319234371185303, 2.252960681915283], [-0.18076768517494202, -0.5899826884269714, -0.3017334043979645, -1.3023154735565186, 0.4825551211833954, 0.018004443496465683, 0.3825001120567322, -0.21087172627449036, -0.5980245471000671, -0.729667603969574, -0.16575518250465393, 1.1000930070877075, -0.08082113415002823, -0.6378429532051086, 1.0122385025024414, 0.6636548638343811, -1.2390024662017822], [0.27031320333480835, 0.339531272649765, -0.6660951972007751, -0.4229958951473236, 0.6071237325668335, -2.2630984783172607, -0.1091398075222969, 0.2216787040233612, 0.8601223826408386, 0.7306676506996155, 0.2398759126663208, 0.6960429549217224, 0.43272969126701355, 0.043149013072252274, -0.24554410576820374, 0.07893019169569016, -0.8928748369216919], [-1.010029673576355, -0.6647199392318726, -0.04412350431084633, 0.23153677582740784, -0.22778457403182983, 1.078963279724121, -0.9533452987670898, -0.48641878366470337, -1.1469752788543701, -0.3970377743244171, -1.4849671125411987, -0.7574072480201721, 0.1483270227909088, 0.26511266827583313, 0.263998806476593, 1.9006290435791016, -1.0811346769332886], [-0.3868606388568878, -0.7638392448425293, -0.5218808650970459, -1.5451730489730835, -1.5478370189666748, -0.3834006190299988, -0.711720883846283, -0.4499889612197876, -1.1998379230499268, -1.3233740329742432, 0.5096018314361572, -1.9122533798217773, -1.4216541051864624, -2.0549845695495605, -0.6115136742591858, 0.19227339327335358, -0.4371773898601532], [0.32626309990882874, -0.2524714469909668, -1.737125277519226, -0.9639498591423035, 1.2982509136199951, -0.5570988655090332, 0.6629133224487305, 0.6989105939865112, -1.400475025177002, -2.3689332008361816, 0.021151963621377945, 0.9282529354095459, 1.000749945640564, 0.5102632641792297, -0.19276879727840424, 0.9076859354972839, 0.04853086546063423], [0.8054419159889221, -0.3752586543560028, 0.3772064745426178, 0.710760235786438, -1.164459466934204, 1.3158150911331177, -1.5497770309448242, -1.0229411125183105, 0.7139126658439636, 3.082087516784668, 0.3306947946548462, 0.9110080003738403, 0.5944429636001587, 0.5486999154090881, -0.34836873412132263, -0.16579565405845642, -1.2419198751449585], [0.7254086136817932, 0.6547418832778931, 0.8652316331863403, -1.2043033838272095, -1.1170799732208252, 0.1407400518655777, -0.6226929426193237, 0.6895214915275574, -0.8709989786148071, -2.429365634918213, 0.3106502294540405, 1.4957281351089478, -2.3568384647369385, 0.44217339158058167, -0.5009677410125732, 0.8724210262298584, -0.2814750671386719], [-0.7177022099494934, -0.10139205306768417, -0.3221971392631531, 0.7879887223243713, 0.8836013674736023, -0.5326441526412964, 2.177013874053955, -1.315961480140686, 1.7910585403442383, 0.13956411182880402, 0.6848642230033875, 0.6427436470985413, -0.9581285119056702, 0.020296216011047363, -1.3310071229934692, 0.12296164780855179, -0.2031441181898117], [0.15244075655937195, -0.09421936422586441, -0.6148778796195984, 0.08587604016065598, -0.7624387741088867, 0.29183948040008545, 0.6342954039573669, -1.1955845355987549, 0.32385894656181335, -1.4861987829208374, 1.9959676265716553, 1.3825608491897583, 0.24392111599445343, 1.0467443466186523, -1.4396495819091797, 0.7966148853302002, -1.1775175333023071], [0.706146776676178, 0.3450494110584259, -0.15197783708572388, -0.3998441994190216, -0.28195202350616455, 0.3109213411808014, 0.2901759147644043, 0.9668812155723572, 0.23474085330963135, 1.3487000465393066, 0.8584679961204529, -0.7388672828674316, -1.1472512483596802, -0.3611181676387787, 0.16219650208950043, 1.4872480630874634, -0.08820318430662155], [0.34460222721099854, 2.3820042610168457, -0.6930063366889954, 0.1663389503955841, -2.4991962909698486, -1.058097004890442, 0.613274335861206, 0.07149188220500946, 0.21155381202697754, 0.7327736616134644, -0.9946842193603516, 0.2354772686958313, 0.030677257105708122, 1.0677976608276367, -0.030478471890091896, -1.8679478168487549, 1.5276758670806885], [-1.1540535688400269, -0.003932910971343517, -2.0714712142944336, 0.4649144411087036, -1.8036315441131592, 1.0888245105743408, -1.3327864408493042, 1.3282536268234253, -0.24572904407978058, -0.2966644763946533, 0.3688831627368927, -0.6629709005355835, 0.793975293636322, -1.4101707935333252, -0.6541810631752014, 1.7822223901748657, -0.2577434778213501], [-0.3717854917049408, 0.006120459642261267, 1.433194875717163, 0.2795911729335785, 0.2668985426425934, 0.433712899684906, 2.281421661376953, -2.048560857772827, 0.5698662996292114, 0.6247674822807312, -0.9726048111915588, -0.07353778183460236, -0.4026041626930237, -0.6751477122306824, -0.4391646683216095, -1.1292269229888916, 0.48152920603752136], [0.5362332463264465, -0.9282805323600769, -0.34741950035095215, 1.5297561883926392, 0.3431999981403351, 2.099792003631592, -0.20936982333660126, 0.1641436666250229, 0.8157685995101929, 0.03752922639250755, -1.534623384475708, 1.49032461643219, -0.17001435160636902, 1.7633013725280762, -0.6591008901596069, 0.535941481590271, -1.4462413787841797], [1.6084928512573242, 0.13490450382232666, -0.1762627214193344, 0.6468929052352905, -1.9710440635681152, -0.8068785667419434, -0.8377702236175537, -0.19685004651546478, 0.25183990597724915, -0.2806759774684906, 0.6814287304878235, 1.4232385158538818, -0.9511440396308899, 1.586280107498169, -1.3906595706939697, 1.326772928237915, -0.09610085934400558], [-1.2356500625610352, -0.08974097669124603, -0.4660508334636688, 0.23309864103794098, -1.4042603969573975, 1.7204267978668213, 0.6143409609794617, -1.0682371854782104, 1.048323392868042, 0.1990872621536255, -0.44614407420158386, -1.8761539459228516, 1.4160088300704956, -1.4617764949798584, 0.6950298547744751, -1.660941243171692, -0.7345489859580994], [-0.832132875919342, -0.5134297013282776, -0.8490562438964844, -0.045128196477890015, -0.21578653156757355, -0.6477997303009033, 0.24628287553787231, 0.06983206421136856, 1.1439261436462402, -0.9305865168571472, 0.04114938899874687, -2.0444982051849365, 1.3784047365188599, 1.9276889562606812, 1.2642630338668823, -1.1638704538345337, -2.215846300125122], [-0.5282120108604431, -0.25421151518821716, -0.04785382002592087, 0.9568506479263306, -0.9637098908424377, -0.3007371425628662, 0.47747930884361267, 0.8251780271530151, -1.4159716367721558, -0.6779423952102661, 0.6094690561294556, 0.599321186542511, -0.5092076063156128, 0.5583654642105103, 0.40853995084762573, -0.9064711332321167, -0.06788770109415054], [-1.1610914468765259, 0.30280816555023193, -0.30420318245887756, 1.0939544439315796, 0.32500404119491577, -0.29568228125572205, -0.9597333073616028, 1.1133209466934204, -0.9724035859107971, -1.9414405822753906, -0.6151180267333984, 0.7328812479972839, -0.20519500970840454, -0.38104695081710815, -1.1724754571914673, 0.6053444743156433, -0.06656306982040405], [-1.2785136699676514, -0.2154879868030548, 0.7464314103126526, 1.5835062265396118, 1.734322190284729, 0.22139793634414673, -1.3279656171798706, -1.4487342834472656, 0.8673388361930847, -1.2221051454544067, 0.08253541588783264, -0.4608916938304901, 1.1063119173049927, -0.36722689867019653, 1.289922833442688, -0.8783848881721497, 0.3545069992542267], [-1.6065913438796997, 0.6669121384620667, 1.0973271131515503, 0.5015744566917419, 0.4769585430622101, 0.7216140627861023, -0.08169999718666077, -1.2305245399475098, -1.0062321424484253, -1.2980031967163086, 0.11021699756383896, -0.3534505069255829, -0.24546155333518982, 0.7243210673332214, -0.560173749923706, -2.460783004760742, 0.3718019425868988], [0.1811520904302597, 0.6719365119934082, -0.42868995666503906, 0.4733148217201233, -0.2799711227416992, -0.8426030874252319, -0.15122449398040771, -0.9603674411773682, 0.5040469169616699, 1.5353920459747314, 0.6216581463813782, 0.9853870272636414, 0.8925021886825562, -1.6253918409347534, -1.8601752519607544, 3.0048351287841797, 0.29593807458877563], [1.013572335243225, -0.7876970767974854, 2.1602530479431152, -0.18602928519248962, -0.7752045392990112, -1.1622997522354126, 0.05853893607854843, -0.717336893081665, 0.39008501172065735, 0.06675708293914795, 1.0167196989059448, 1.4737764596939087, -0.1298556625843048, -1.1049391031265259, -0.3607556223869324, -0.22997401654720306, -1.2786617279052734], [-1.0806621313095093, 1.8178812265396118, -1.4582676887512207, 0.3710083067417145, 0.25536638498306274, 0.9483757615089417, -1.3840490579605103, 0.2632509768009186, -0.337040513753891, 1.0924654006958008, -0.8648908734321594, 0.4354041814804077, -0.4722876250743866, 1.301242470741272, 0.1647668182849884, -1.671745777130127, 0.49347472190856934], [0.1787911355495453, -0.5257353782653809, 0.006966217886656523, -0.269408255815506, 1.5122220516204834, 0.523769199848175, 1.168510913848877, -1.1354314088821411, -0.534917950630188, 1.618714451789856, -0.74267578125, 0.8546227812767029, 0.19815754890441895, 1.25819993019104, 1.0295807123184204, 0.19130580127239227, 0.7666262984275818], [-1.0023584365844727, 1.1309598684310913, 0.5163525342941284, -0.04857408627867699, 0.3748786747455597, -0.5824365615844727, -0.5864565968513489, -1.1904376745224, 0.4402509927749634, 0.1004631370306015, 0.9284665584564209, -0.6022985577583313, -0.7292144298553467, 0.7188222408294678, 0.23380827903747559, -1.0603678226470947, 1.70330011844635], [1.8520256280899048, 0.2509163022041321, -0.4583984613418579, -0.3915698230266571, 0.8459693193435669, -0.7901849746704102, 1.018738031387329, -0.9178052544593811, 1.3104184865951538, -0.18707475066184998, 0.9174692034721375, -0.7690924406051636, -2.629450559616089, -0.6941364407539368, 0.4733896553516388, -0.37085476517677307, -0.9032139182090759], [0.06118423119187355, -0.3399754762649536, 1.6339128017425537, 0.9581788182258606, -0.02255067601799965, 0.34824398159980774, 0.5015763640403748, 1.6465352773666382, -0.6856637001037598, -0.5473641157150269, -0.6458963751792908, -0.44706273078918457, 0.6115155220031738, 1.8828717470169067, -0.8925445675849915, -0.8205267786979675, 0.48086559772491455], [0.6754019856452942, 2.5321853160858154, 1.3374983072280884, -2.0794975757598877, 1.8633580207824707, -1.0877387523651123, -1.1620286703109741, 2.2274739742279053, 1.976940631866455, 0.7064495086669922, 0.5875034332275391, -2.163045644760132, 1.0081125497817993, -0.2727099359035492, -0.027431122958660126, -0.9202880859375, -0.2606711983680725], [-1.1189385652542114, 0.6795072555541992, -2.323418140411377, -1.3308385610580444, -1.0058379173278809, 0.861399233341217, 0.6364497542381287, -0.4902629852294922, -0.20040051639080048, 1.1672556400299072, 1.1037510633468628, 1.1107169389724731, 0.7879657745361328, 0.25188928842544556, 0.6158168911933899, -0.9779224991798401, 0.18584974110126495], [1.3892587423324585, -0.6858532428741455, -1.5330222845077515, 1.1989823579788208, -0.3230211138725281, 0.2585121691226959, -0.7270907759666443, 1.0801732540130615, 1.9145028591156006, -0.7475360631942749, -1.8058242797851562, -0.576424241065979, -0.5744298696517944, 0.4800371825695038, 0.642144501209259, -0.73818039894104, -0.3606449067592621], [0.2997804284095764, 0.10833559930324554, 0.748033881187439, 1.3383150100708008, 1.3030643463134766, 0.27526697516441345, -0.13545696437358856, -0.7307266592979431, -0.4114096760749817, 0.7696714997291565, 1.3680740594863892, 0.11620963364839554, 0.8420232534408569, -0.7365379333496094, -2.2709829807281494, -0.23007066547870636, 0.7303726673126221], [0.0688319280743599, 1.1042097806930542, -0.6411383748054504, -0.16991592943668365, 0.18191353976726532, 0.3899677097797394, 0.48640555143356323, 0.26185867190361023, 0.3362191915512085, 1.008750557899475, 0.1366751343011856, -0.1054864227771759, -1.3665716648101807, -0.09827913343906403, 1.7744135856628418, 1.6645770072937012, -0.06575030088424683], [0.9483299255371094, -0.8089126348495483, 0.45413514971733093, -1.2445435523986816, -0.43550872802734375, 0.05841570347547531, 0.32064998149871826, 0.059617288410663605, 0.5173310041427612, 1.8120293617248535, 0.7242456078529358, 0.19949141144752502, 0.4123125374317169, -0.9386307597160339, -0.19313740730285645, 0.80774986743927, 0.2597343623638153], [1.9533758163452148, -0.2711014449596405, 1.9670891761779785, -0.5452799201011658, -0.19513556361198425, -0.23260076344013214, -0.379540354013443, 0.14946380257606506, 0.7579165101051331, -0.558858335018158, -1.1708792448043823, -0.7291480302810669, 0.9687039256095886, -0.4689962863922119, 0.6148732900619507, 1.37267005443573, 1.1112993955612183], [0.7276427745819092, -0.7688127756118774, -1.1720198392868042, 0.09286101162433624, 0.18041445314884186, 1.2151070833206177, 0.8989284038543701, -0.09999172389507294, 1.8334548473358154, 0.470302015542984, -0.6943142414093018, -0.5585904121398926, -1.2059335708618164, -0.2788666784763336, -0.6615521311759949, 0.5174208879470825, 1.6889370679855347], [0.16013038158416748, -0.1836138218641281, 0.8931760191917419, 0.3046477735042572, 0.16507822275161743, -0.009904770180583, 0.07182291895151138, 0.1595100313425064, 0.07557693868875504, -0.7904220819473267, -0.2925386428833008, 1.281061053276062, -0.9643306732177734, -0.4174017906188965, -0.06298431009054184, 1.4073874950408936, 1.772208333015442], [-2.39121675491333, -0.07822463661432266, -0.04900984466075897, 0.7377126812934875, 0.19423498213291168, 1.1984034776687622, 1.3878475427627563, 1.4421297311782837, 0.2961174547672272, 0.7137188911437988, 1.2967575788497925, -1.3190178871154785, -0.1145339161157608, 0.08221914619207382, -1.1772041320800781, -0.7894790768623352, 1.3624370098114014], [-1.588918685913086, 1.2305214405059814, 0.569347620010376, 0.13468915224075317, -0.782242476940155, 2.8609731197357178, 0.899131715297699, 0.3672027885913849, 0.08955998718738556, 0.9187162518501282, -0.0639728233218193, 1.1882572174072266, -0.3507237434387207, 0.41232380270957947, -0.8241257667541504, -1.7596081495285034, -0.2519094944000244], [0.8549396395683289, -0.7245749235153198, 0.3658069670200348, 1.8714463710784912, 0.8617430329322815, 1.1159857511520386, -0.8900346755981445, 1.9173222780227661, -0.8934804797172546, 0.36412638425827026, -0.30755680799484253, -0.15101036429405212, 0.35712432861328125, 2.3431572914123535, 1.1946206092834473, -0.6584307551383972, -1.6164401769638062], [-0.3888627588748932, 0.13788050413131714, -1.9750685691833496, 0.39396169781684875, -0.23846592009067535, 0.6050673723220825, -1.166176676750183, 0.172738716006279, 0.18619714677333832, 1.6956267356872559, -0.21963389217853546, -0.08859249949455261, -0.422783225774765, 0.08206906914710999, -0.39152202010154724, -1.5191736221313477, -1.378440022468567], [-0.6251206398010254, -0.759951651096344, -0.3923766613006592, 0.6460699439048767, -1.431739330291748, 1.2756071090698242, 2.0832297801971436, -0.30563026666641235, -2.555706262588501, 0.060669731348752975, 0.3220982551574707, -1.7388290166854858, 0.9277763366699219, -0.07632467895746231, 0.5016145706176758, -3.6929306983947754, -0.469845175743103], [0.6977530121803284, 1.287442684173584, 0.025391098111867905, 1.2767928838729858, 0.6598103046417236, -0.25640857219696045, 0.8935390710830688, -0.06074916571378708, 2.068424701690674, 1.312109351158142, 0.05901646986603737, -0.5598410367965698, 0.5191088318824768, -0.23196223378181458, 0.1351793259382248, 1.4939712285995483, -0.915829598903656], [-1.3311223983764648, 0.5030822157859802, 2.4966537952423096, -0.5535812377929688, 1.7438234090805054, -0.20142966508865356, 0.9754732251167297, 0.0245005264878273, 0.9851717352867126, -1.047673225402832, -1.6280704736709595, -0.06400775909423828, 0.7152875661849976, 0.04670615866780281, 1.8850713968276978, 1.0029881000518799, 0.9307969808578491], [0.02626148611307144, 0.19565829634666443, 0.2883371412754059, -1.323099136352539, -1.1084047555923462, -1.4754282236099243, 1.0479472875595093, 0.5614673495292664, 0.8997523784637451, 2.4781901836395264, -0.34832563996315, 1.3916010856628418, 0.5779988765716553, 0.002581241773441434, 0.5649282336235046, 0.40489432215690613, -0.08385398983955383], [0.7977298498153687, 1.2467679977416992, -1.023675560951233, -0.27916598320007324, 1.198040246963501, -0.18242792785167694, 0.9286947846412659, 1.3876752853393555, 0.3375086486339569, 1.4612574577331543, -0.9410952925682068, 0.23758357763290405, -1.4912471771240234, -1.4987210035324097, -0.37996989488601685, -0.23355594277381897, -0.15806356072425842], [0.08493481576442719, -0.3348006010055542, -2.1240909099578857, 2.0824966430664062, -0.7394412755966187, -0.4660511612892151, -1.2787940502166748, -0.2641918361186981, -0.977059006690979, -1.3518033027648926, 1.6748749017715454, -0.42223480343818665, -1.9090853929519653, -0.4753533601760864, 0.8359180688858032, 2.023423433303833, -0.7066839933395386], [-0.9437429904937744, 0.255005419254303, 0.20377005636692047, -0.8658243417739868, -1.4703724384307861, -0.169785737991333, 0.1553475707769394, -1.3410574197769165, 0.9019075036048889, 0.08622699975967407, 1.1630547046661377, 0.745804488658905, -0.7872406244277954, 0.17353789508342743, -0.7767148613929749, 0.17015759646892548, -0.6967443227767944], [-1.4673359394073486, -0.006223725620657206, -1.3655918836593628, -2.284430503845215, 1.1940900087356567, 1.5365030765533447, 1.1919541358947754, 1.3687596321105957, 0.6671657562255859, -1.6340818405151367, 0.44747069478034973, 0.6356918215751648, -0.7758427858352661, 0.40061357617378235, 1.390376329421997, 0.8859560489654541, -0.7272250056266785], [0.918421745300293, -0.4977918565273285, 0.326291024684906, -0.5008091926574707, 0.1485864222049713, -0.8241773247718811, -0.2337189018726349, 0.34697699546813965, 0.7728905081748962, 0.13487759232521057, 0.7230033278465271, 0.09940443933010101, 0.4246751070022583, 0.37738001346588135, -0.013277556747198105, -1.311556100845337, 0.13378284871578217], [0.4997619092464447, -0.14752477407455444, 0.3693047761917114, -0.3385705053806305, -0.1007814109325409, 1.1868494749069214, 0.22293618321418762, 0.10436976701021194, 0.1442042589187622, 0.6722369194030762, -0.2980164587497711, 0.5681706070899963, -0.8692265152931213, 1.3432941436767578, -0.3881242871284485, -0.17465654015541077, -0.27182093262672424], [-0.9001033306121826, 0.05424464866518974, -0.08468149602413177, 0.3002062141895294, 0.8995680809020996, -0.5598393082618713, 0.20378416776657104, 2.5472962856292725, 1.0652403831481934, 0.5491710901260376, -2.2999110221862793, -2.229279041290283, 0.32785823941230774, -1.655789852142334, 0.24122150242328644, 0.600350558757782, -0.020171809941530228], [-0.32833731174468994, -0.8152564764022827, 0.05385415628552437, -0.7812987565994263, 0.2450987994670868, -0.6587417125701904, 0.9375187754631042, -0.4720308482646942, -0.7846558690071106, -0.6378848552703857, 0.21374733746051788, -0.34195518493652344, -1.6668691635131836, -0.1982121467590332, 0.49996471405029297, -1.247114896774292, 0.007595052942633629], [1.1012036800384521, -0.22588753700256348, -0.2915925085544586, -0.45355042815208435, -0.5239288806915283, -0.9475558400154114, 0.3512209951877594, -0.9240179061889648, -0.2732119560241699, -0.6248686909675598, -0.2723219692707062, -0.9376937747001648, -1.6170382499694824, 1.0855469703674316, 0.8578344583511353, 1.6589853763580322, -0.17134274542331696], [0.15711131691932678, -0.4068294167518616, -1.3120577335357666, -0.7475840449333191, -0.598842442035675, -1.407591700553894, 0.4221518635749817, 0.057282835245132446, -0.578894317150116, 1.7548023462295532, 0.6814409494400024, -0.26422086358070374, 1.1557079553604126, 1.3894600868225098, -1.5983591079711914, -1.994216799736023, -1.9771194458007812], [-0.6265634894371033, 0.45642125606536865, 0.042216308414936066, 0.5732845664024353, -1.150342345237732, -1.4877171516418457, -0.9016847014427185, -0.15368156135082245, -1.093971610069275, -0.20931696891784668, 0.16606077551841736, 1.4372711181640625, -0.2488963007926941, -0.742716372013092, -2.2180585861206055, 0.21891960501670837, 0.5555514097213745], [-2.083824396133423, 0.46044379472732544, 0.2960197627544403, 0.8945165276527405, -1.8428919315338135, 0.7320181727409363, 0.7306117415428162, -0.9036141633987427, 0.7320507764816284, -1.3710769414901733, 0.22145827114582062, -1.7751628160476685, -0.5134789347648621, -0.20342093706130981, -0.16805097460746765, -0.7423138618469238, -0.4478706419467926], [0.3219010829925537, 0.3695077896118164, -0.15894971787929535, 0.13871002197265625, -0.8064540028572083, 0.8549301624298096, -1.1481947898864746, -0.34473711252212524, 0.18976564705371857, 1.9608347415924072, 0.3225845992565155, 0.7578925490379333, 0.1543530970811844, -0.6189851760864258, -0.01973971724510193, -2.6024696826934814, -1.9679460525512695], [1.3928673267364502, 0.13437187671661377, 1.369279384613037, -0.06689618527889252, 0.30183538794517517, 1.3612743616104126, -0.4223518967628479, 1.012297511100769, -0.3413764536380768, 0.2818470597267151, 0.2973063886165619, -0.31125974655151367, -1.13223135471344, 0.18279190361499786, -1.1618711948394775, -1.0445841550827026, 0.9026418924331665], [-0.42145687341690063, 0.5748171806335449, 0.4992658793926239, 0.3872390389442444, -0.20741739869117737, -0.007371796295046806, 0.31133827567100525, 0.3019494116306305, -0.5236497521400452, 0.46475908160209656, -1.8431214094161987, -1.1911723613739014, -0.3870491087436676, 2.453917980194092, 0.46462082862854004, 1.0869275331497192, -0.20644007623195648], [0.5986878275871277, -0.5073113441467285, -0.8708054423332214, -0.32947084307670593, 0.18701724708080292, 0.20370179414749146, -0.049106717109680176, 1.2862416505813599, 0.7990225553512573, -1.5831283330917358, 0.5639451742172241, 0.004805141128599644, 0.23541545867919922, -0.40100210905075073, -1.5146571397781372, -2.135603666305542, -0.8150202035903931], [1.5774612426757812, -2.0130577087402344, 0.30966317653656006, 1.0734020471572876, -0.8789721727371216, 3.066567897796631, 1.0362703800201416, -0.16285353899002075, 1.090777039527893, 0.12602286040782928, -0.9010083079338074, 1.2363065481185913, -2.5545670986175537, 0.6033196449279785, -0.0941954106092453, -2.0701539516448975, 0.36605408787727356], [-0.026593917980790138, 1.108667016029358, 1.8811190128326416, 0.36070793867111206, 0.3351975381374359, 0.9074715375900269, -0.7245786786079407, 0.35133713483810425, -0.20517531037330627, 0.14972124993801117, 1.394526720046997, 1.302162528038025, 1.0619182586669922, -1.3639763593673706, 0.6162497401237488, 0.4928629398345947, 0.2519664764404297], [0.7461109757423401, 0.8275274038314819, 1.492854118347168, -0.9322457313537598, 1.5946348905563354, 0.8847583532333374, -0.07027579098939896, -0.10911156237125397, 0.3354772925376892, -1.3224200010299683, -0.7782095074653625, -0.032424446195364, 0.23786190152168274, -0.3437250256538391, -0.12262129783630371, 2.6702916622161865, -0.8142878413200378], [0.171543687582016, -0.843227744102478, -2.103959083557129, -1.242289662361145, 0.5564391613006592, 1.8411532640457153, -1.531323790550232, 1.5291942358016968, -0.191973477602005, 0.36824122071266174, -1.2405261993408203, 0.21674291789531708, 0.3220907747745514, -0.8183559775352478, -0.08212074637413025, 0.040901534259319305, -2.583533525466919], [0.6846983432769775, -0.6841850876808167, -0.39932873845100403, -1.0756802558898926, 1.3434433937072754, -1.0744259357452393, -1.3378896713256836, -1.9839342832565308, -0.6893115639686584, -1.291195034980774, 0.05207168683409691, 1.1539099216461182, -2.1209988594055176, -1.2714693546295166, -0.6333024501800537, 0.9733680486679077, -2.6295249462127686], [-0.5256322026252747, 0.023247716948390007, 0.4861462414264679, -0.5947040319442749, 2.3719470500946045, -0.9328453540802002, 0.6103687286376953, 0.7325407266616821, -1.2379118204116821, -1.045159101486206, 1.0633519887924194, -1.2819310426712036, 0.4577077031135559, 0.3433799743652344, 0.969419538974762, 0.38712888956069946, -0.1725919097661972], [1.042236089706421, 1.2488598823547363, -0.9239414930343628, -1.2816030979156494, -0.41505083441734314, 0.9889249801635742, -1.2950005531311035, -0.8070540428161621, -0.67708820104599, -1.8193650245666504, -1.1045482158660889, 0.8201137781143188, 1.8493753671646118, -1.6338499784469604, -2.209085702896118, 0.1663554310798645, 0.1493857353925705], [-0.987946093082428, 0.32650652527809143, -0.8006755113601685, -0.09887005388736725, 0.791199803352356, -0.06623655557632446, 1.6591780185699463, -1.6176444292068481, -0.5112835764884949, -1.5365145206451416, 0.14605006575584412, 0.9676942825317383, -0.04351061210036278, -0.7098431587219238, -0.41220271587371826, 0.03255775570869446, 1.9731899499893188], [-0.017403578385710716, -0.09171337634325027, 0.05974140763282776, 0.3898950219154358, 2.1183955669403076, -0.40962085127830505, 0.7792675495147705, 0.6285161375999451, 0.020549332723021507, 2.119154453277588, -1.418482780456543, 0.1793569177389145, 1.4611845016479492, 0.4990786910057068, 1.3164856433868408, 0.21147815883159637, 0.8062101602554321], [0.7100993990898132, 0.5582063794136047, -0.8614655137062073, -0.05951283499598503, 1.1761192083358765, 0.2470615953207016, -2.165351629257202, -0.05472038313746452, -0.6639230847358704, 1.785914659500122, -0.7862390279769897, -0.223915696144104, 0.5559415817260742, -0.3066868782043457, -1.8415586948394775, -0.03374187648296356, 0.9990503191947937], [0.4946419298648834, 0.4613938629627228, 0.8126850724220276, 0.30611452460289, 1.1226475238800049, -0.4645170271396637, -0.48564237356185913, 0.732916533946991, -0.770557701587677, 0.2848812937736511, -0.0779045820236206, -1.3489007949829102, -1.66342031955719, 1.4696472883224487, 0.19983507692813873, 0.23798659443855286, -0.2215181440114975], [1.7843315601348877, 0.26735255122184753, -0.5613421201705933, 0.2736057639122009, 0.8278509974479675, -0.6387161612510681, 1.4595136642456055, 1.356100082397461, -1.3042194843292236, 0.24494439363479614, -1.9178733825683594, 0.243050679564476, 1.6107068061828613, 0.9025300145149231, -1.2849465608596802, 0.7892489433288574, -0.7047424912452698], [1.1340450048446655, 1.4932299852371216, -0.513413667678833, -0.3397941291332245, 2.764259099960327, -0.9837594032287598, 0.07764692604541779, 0.9841787815093994, 0.38750043511390686, -0.7664498686790466, -1.1326044797897339, -0.22584949433803558, -0.44224682450294495, -0.9026394486427307, 0.554908812046051, -0.6429433822631836, -0.33235371112823486], [-0.07243875414133072, -0.6839310526847839, 1.278072714805603, -0.3392828404903412, -0.5133673548698425, -0.18284974992275238, -0.2256583869457245, -1.528460144996643, 0.32154613733291626, 0.3341536223888397, 0.15595196187496185, -1.3179280757904053, 0.9182488918304443, 0.7353853583335876, 1.11969792842865, 0.4798911511898041, 1.471093773841858], [-0.44548529386520386, 0.05443914234638214, -1.2383345365524292, -0.501412034034729, 1.2525826692581177, -2.2608630657196045, -0.9798951148986816, 0.2286473959684372, 0.6162548661231995, -0.41063329577445984, 0.24256232380867004, 0.4694160223007202, -0.6187198162078857, -0.03566548228263855, -1.0533782243728638, 0.39738741517066956, -0.8192929029464722], [-0.9103994369506836, 0.38603854179382324, 0.11617270112037659, 0.10010653734207153, -0.4462531805038452, 0.9534225463867188, 1.0609793663024902, 0.2519564926624298, 0.3246516287326813, 0.2021200805902481, 0.21802611649036407, 0.5689175128936768, 1.0127350091934204, -1.8670786619186401, 0.1839209794998169, -0.6209297776222229, -0.4536000192165375], [0.2627573609352112, 1.2689838409423828, -0.2375032603740692, -0.4580467641353607, 2.8805625438690186, -0.8034341335296631, -0.05651037395000458, -0.2316257804632187, 0.4772009253501892, -0.6884580850601196, 1.1400753259658813, 1.0249336957931519, -0.15926171839237213, 1.2733972072601318, 0.30463385581970215, -0.31074246764183044, -0.21893253922462463], [-0.9898427724838257, -1.0375398397445679, 0.1619383990764618, 1.755824327468872, -0.15677510201931, 1.0987639427185059, -0.5982592701911926, 1.1773583889007568, 1.513841986656189, 0.8505690693855286, -1.3133578300476074, 0.32985997200012207, 0.02666228450834751, 1.4620789289474487, -0.3689856231212616, 0.2148531824350357, 0.11848442256450653], [0.09298580139875412, -0.2732193171977997, -0.2900673747062683, 1.3827146291732788, -0.8826257586479187, 0.7889521718025208, -0.38797131180763245, 0.11152727156877518, 0.8982279896736145, 0.17481127381324768, -0.19789312779903412, -0.45972806215286255, 0.4263826906681061, -0.2709537148475647, 0.6797360181808472, -0.09378191083669662, -1.1710913181304932], [-0.657489538192749, -0.19860951602458954, -0.3731081187725067, 2.2247421741485596, -1.5422471761703491, -0.16936621069908142, 1.7039622068405151, -0.19841623306274414, -0.2627744674682617, 0.19286204874515533, 0.05905504897236824, 0.4406099319458008, -1.0866029262542725, 0.436847984790802, 0.5360758900642395, -0.5616902709007263, -0.5122727155685425], [-0.44106847047805786, 0.3975359797477722, 0.6496263742446899, 1.597267508506775, 1.2096668481826782, 0.8421158790588379, -0.6422317028045654, -0.3852182924747467, 0.3711799681186676, -0.9845284223556519, 1.4555845260620117, 1.7553009986877441, 0.6376194953918457, -0.8019090294837952, 0.5032550096511841, -0.5662864446640015, 0.033226799219846725], [-0.3513299524784088, -0.5517538189888, -0.36827725172042847, 0.5077763795852661, 1.153923749923706, 1.3511278629302979, -0.8313790559768677, 1.005401611328125, -0.5994210839271545, -0.6367322206497192, -0.28277263045310974, 0.06655286252498627, 1.551182508468628, -1.169311285018921, 0.9616504907608032, -0.7231795787811279, 1.664498209953308], [-0.8441780805587769, 1.1318355798721313, -1.2034976482391357, -0.3330264091491699, -1.0262750387191772, -0.7257192134857178, -0.21152466535568237, -0.6039078235626221, 0.5569080114364624, -0.9783310890197754, -1.068405270576477, 1.1857774257659912, -1.5211412906646729, 0.31283432245254517, -0.28863513469696045, 1.1719859838485718, -0.1443643867969513], [-2.090705633163452, -1.8783539533615112, 1.1247310638427734, -0.030512487515807152, 1.5734355449676514, 2.333747625350952, 0.9873639941215515, 0.0705510601401329, 0.3980795443058014, 0.48163533210754395, 0.49272364377975464, -0.23975451290607452, -0.9771351218223572, 1.3216817378997803, -0.36407631635665894, 0.3930647671222687, 1.2718722820281982], [-0.6657086610794067, -0.5074735283851624, 0.9401954412460327, -0.8663022518157959, -1.167734146118164, -0.09553886204957962, -0.18618720769882202, -0.8090444803237915, -0.7614671587944031, -1.1440531015396118, 0.3487548530101776, 1.6057499647140503, 0.22849823534488678, -0.33421483635902405, -0.18862982094287872, 0.7317339181900024, 1.9442956447601318], [0.5235700011253357, 0.06626497954130173, 0.7278583645820618, 0.3054090440273285, 0.34728893637657166, 0.23929941654205322, 1.239986538887024, 0.23512256145477295, -0.9866324663162231, -0.6385877728462219, -1.361240029335022, -0.35495391488075256, -1.095131516456604, -0.2595178782939911, 3.334350347518921, -0.7099891304969788, 0.04264231398701668], [-0.9940730929374695, 1.0635958909988403, -1.8074759244918823, 0.25712889432907104, -0.4762248992919922, 1.6459463834762573, 1.3165411949157715, -0.36748766899108887, 0.47917962074279785, 0.10477221757173538, 1.9630664587020874, 0.03183361515402794, 0.3241042196750641, -0.44497331976890564, -1.244459867477417, 0.04175712168216705, -0.9802802801132202], [0.3354583978652954, -1.2598645687103271, -0.3451586663722992, -0.28978151082992554, 0.6108564734458923, 0.41214877367019653, -0.933073878288269, -0.17029869556427002, -0.55759596824646, 0.5525276064872742, 0.1711675524711609, -1.1646798849105835, 0.09251882880926132, 0.5612539649009705, -0.4107602834701538, -0.13768306374549866, 0.7494952082633972], [-0.5444778800010681, -0.709883987903595, 0.09027313441038132, -0.6635324358940125, -1.8103001117706299, -0.3377051055431366, -0.12740914523601532, 2.0731523036956787, -0.48522523045539856, 0.460292249917984, -2.052438735961914, 0.9978845119476318, -0.9252890348434448, -0.6562081575393677, 0.3544791042804718, -0.3864199221134186, -1.3207391500473022], [-0.9808117151260376, -0.007216454017907381, -0.3628842532634735, -1.0745514631271362, 0.8614660501480103, 1.3779840469360352, -0.7243924140930176, 0.2617110311985016, 0.11378059536218643, -1.4327564239501953, -1.3965115547180176, 0.7326652407646179, 0.8228740096092224, 0.29753050208091736, -0.6435865759849548, -0.2934788763523102, 0.9586524367332458], [-0.010096202604472637, -1.1844004392623901, -0.36956629157066345, -0.34365323185920715, -0.28692665696144104, 1.323242425918579, -0.2031364142894745, -1.2184975147247314, 0.5458790063858032, -0.6229453682899475, 1.2280484437942505, -0.031244877725839615, -0.73869389295578, 0.032791268080472946, 0.26974058151245117, 0.11163902282714844, -0.8074893355369568], [0.4415946900844574, -0.6224504709243774, -2.0595381259918213, 0.498006135225296, -0.6217693090438843, -1.4103713035583496, -0.7010727524757385, 0.22691629827022552, -2.0993080139160156, -1.6777763366699219, 1.3693113327026367, 0.2235112339258194, -0.9087190628051758, -0.7298588156700134, 0.2883906662464142, 2.202420473098755, -0.7870411276817322], [0.6266550421714783, 0.5945568084716797, -0.9486420750617981, -1.5855382680892944, 1.1714909076690674, 0.1845184862613678, 0.7969215512275696, -2.6831274032592773, 1.6859301328659058, -1.140404462814331, 0.12694419920444489, 0.6353789567947388, -0.5961335897445679, -1.0877653360366821, -1.7226898670196533, -0.4830317497253418, -1.4560065269470215], [-0.16036531329154968, -0.18854765594005585, -0.6566278338432312, -0.2694246470928192, -0.8345805406570435, -0.1842392534017563, 0.7381767630577087, -2.3872570991516113, 0.0793856680393219, 0.837490975856781, -1.4672707319259644, -0.1737225353717804, 0.46556994318962097, -1.45273756980896, 1.5689493417739868, -1.4034783840179443, 0.03351708501577377], [-0.7846041321754456, -0.19289880990982056, 1.332763910293579, -1.433691382408142, -0.6981390118598938, -1.157448649406433, 1.1596581935882568, -1.349724292755127, 1.1868122816085815, 0.7002151608467102, -0.2501090168952942, -0.9887678623199463, -0.4694167375564575, -0.7680371403694153, -0.7576443552970886, 0.7700079679489136, -0.7777526378631592], [-0.2171594351530075, 1.0337527990341187, 0.1318819522857666, -0.20939931273460388, -1.370466709136963, 1.459040880203247, -1.318947672843933, -0.7301973104476929, -2.1819374561309814, -0.9851998686790466, -0.6519084572792053, -2.6138031482696533, -1.2175631523132324, 0.5191912651062012, -0.23094774782657623, 0.679262638092041, 0.40776991844177246], [0.7338775396347046, -0.20778457820415497, 0.8536529541015625, 2.461090326309204, -0.08532349020242691, 0.16357608139514923, 1.5878546237945557, -1.6966972351074219, -1.6092844009399414, 0.9019497632980347, 0.2776230275630951, 1.35005521774292, 1.6004136800765991, 1.5430644750595093, -0.10169384628534317, 1.821117639541626, 0.37962356209754944], [1.1387306451797485, -0.41796350479125977, 0.2157125174999237, 0.491344690322876, -1.1620330810546875, -1.4898303747177124, -1.7937946319580078, -0.4406049847602844, -0.8763788342475891, 0.7549229860305786, -0.38679832220077515, -1.7770259380340576, -0.6233577728271484, -0.6954362988471985, 1.0773621797561646, -0.5761113166809082, 0.032415904104709625], [1.5154376029968262, 0.9345346689224243, -0.5812153220176697, -0.5974259972572327, 0.6981945633888245, -0.5609132647514343, 0.3178844749927521, 2.0297627449035645, -0.30386117100715637, 0.10874823480844498, 1.7448018789291382, 0.3627697825431824, 0.15383777022361755, 1.7787485122680664, -0.026741178706288338, 1.2299264669418335, -1.94767165184021], [1.2013882398605347, 0.18883530795574188, 0.9940367341041565, -1.0148727893829346, 0.766830325126648, -0.9919135570526123, -0.6393229961395264, 1.5654932260513306, -0.12085890024900436, -0.27954497933387756, 1.6025127172470093, -0.5050187706947327, 0.13950811326503754, -0.8642981648445129, 0.5780083537101746, 0.6548126935958862, 0.435596764087677], [0.6001514196395874, 0.07744694501161575, -1.0715408325195312, 0.19492444396018982, 1.1671713590621948, -1.54853355884552, -0.2022750973701477, 0.8890725374221802, 0.08857309073209763, -1.8733084201812744, 0.10347076505422592, 0.46269479393959045, 0.39655229449272156, -0.12316160649061203, 0.16585713624954224, -0.7407686114311218, 2.972564220428467], [0.1240934357047081, -0.9106844663619995, 0.8982526659965515, 0.7340685725212097, -0.533578634262085, 1.0341182947158813, -1.0069403648376465, 1.2353676557540894, 1.0975844860076904, 1.015541672706604, 1.6178629398345947, -1.1380196809768677, -0.3068963885307312, -1.7127447128295898, -0.7239916324615479, -1.0341572761535645, 0.9336735606193542], [-1.268624186515808, -0.39778581261634827, 1.1416194438934326, 0.600368320941925, 0.9179835319519043, 1.523701786994934, 0.06189380958676338, 0.3248269259929657, 0.19745072722434998, 0.8867114186286926, -1.3421043157577515, -0.2695506513118744, 0.11945081502199173, -2.394517660140991, -2.2988452911376953, -1.217327356338501, 0.3849644362926483], [0.4008757174015045, 2.6209635734558105, -2.305293083190918, 0.09054684638977051, -0.6719292402267456, 1.4770870208740234, 0.28950849175453186, -2.2192952632904053, -1.0956156253814697, -1.313100814819336, 0.2670380175113678, 0.17293724417686462, -1.7214909791946411, 0.32713374495506287, 0.3060421347618103, 0.7618379592895508, 1.0358221530914307], [-2.0300419330596924, 0.6516313552856445, 0.08656889945268631, -1.1472406387329102, -0.21645766496658325, 0.38016819953918457, 0.9824903607368469, 0.7950001955032349, -0.5329343676567078, 0.2478789985179901, -0.34952405095100403, -1.2211048603057861, -1.128403663635254, 0.14400163292884827, 1.652633786201477, 0.7161356210708618, 1.4225281476974487], [0.974907636642456, 1.0499756336212158, 2.654148817062378, -1.3198034763336182, -1.018332839012146, 1.6399363279342651, -1.0478705167770386, 1.0382177829742432, -1.0153180360794067, -0.3565092384815216, -0.6458998322486877, 0.13628867268562317, 0.6084731221199036, -1.9056531190872192, -0.2910064458847046, 0.682066798210144, -0.5210800766944885], [-0.2978772819042206, 0.9438368678092957, 1.5009015798568726, 0.04041117802262306, -0.42957907915115356, 1.3672516345977783, -0.6912670135498047, 0.03624185174703598, 0.5757038593292236, -0.8232926726341248, 1.6794917583465576, 1.019757628440857, 0.7785373330116272, -0.28816550970077515, -0.9889290928840637, -0.2739487886428833, -0.1364193558692932], [0.5381677150726318, -1.590084433555603, 0.6195865869522095, -0.4883902668952942, 0.9872292280197144, 0.2773946523666382, 1.2207711935043335, -0.4251559376716614, 0.18373553454875946, -0.42107945680618286, 0.15042968094348907, 1.2743297815322876, 0.8297935724258423, 1.4189456701278687, 0.2760655879974365, -0.3112782835960388, 1.7634813785552979], [-1.6494680643081665, -0.6526139974594116, -0.0092236939817667, 0.12134633958339691, 2.354450225830078, 0.23803162574768066, -0.7543070912361145, -2.2562360763549805, -0.5798705220222473, 0.3767600655555725, -1.5824521780014038, -0.7858291864395142, -1.6415079832077026, -0.08569955825805664, -2.965400218963623, -1.0820993185043335, 0.346195787191391], [1.104241132736206, 0.7264670729637146, -0.20138052105903625, 1.3722914457321167, 0.3597249388694763, 0.1196809634566307, 1.6964325904846191, -0.67332524061203, 0.2726775109767914, -0.6470462083816528, -1.1604301929473877, 0.06645569205284119, -1.1905094385147095, -1.1312764883041382, -0.9233558773994446, -0.6936547756195068, 0.13079530000686646], [-0.6778672337532043, -0.8073782920837402, -0.23008666932582855, -0.750167191028595, -0.22920668125152588, -1.518845796585083, -0.11226043850183487, 0.16955293715000153, -1.151594638824463, -0.06943368166685104, 0.7259440422058105, 0.7251736521720886, -0.6378052234649658, -0.035043079406023026, -2.154057741165161, -1.9009759426116943, 1.5280364751815796], [-0.23636290431022644, 1.3877583742141724, -1.730916976928711, 0.8225193619728088, 1.0668361186981201, -0.21835587918758392, 2.0961546897888184, -2.196462631225586, -1.6981663703918457, 1.492474913597107, 0.1977459043264389, 1.1851811408996582, 1.1760541200637817, 1.7312391996383667, -1.386020541191101, -0.4829837381839752, -1.5536726713180542], [0.6934903860092163, -0.0403718426823616, 0.9041181206703186, -0.4886248707771301, 0.9133931398391724, -0.8593697547912598, 0.18315783143043518, -2.084651231765747, -0.16128483414649963, -2.4778621196746826, 0.4466981887817383, 0.15115399658679962, -2.0197627544403076, 1.3738155364990234, -1.0374171733856201, -1.0713813304901123, -1.4816890954971313], [0.0032550001051276922, -0.8643621802330017, -0.530792236328125, 1.3099074363708496, -1.8135658502578735, 0.5623325705528259, 0.7179291844367981, 0.07476165890693665, 1.2139662504196167, -0.22105053067207336, 0.6912184953689575, 0.0019370397785678506, -0.08484231680631638, 0.22054508328437805, 0.3635272681713104, -0.36512067914009094, -0.050079721957445145], [-1.9784585237503052, 0.34608617424964905, 1.1267285346984863, 0.31141817569732666, 0.274086058139801, -0.5802344083786011, 1.3271459341049194, -1.1789315938949585, -0.5317499041557312, 1.6770472526550293, 0.9511245489120483, -0.263923317193985, 1.4110177755355835, 2.091273069381714, -0.636812150478363, 2.1092963218688965, 1.9071677923202515], [0.03935512527823448, 0.7535161972045898, -0.9767951369285583, -0.3208877742290497, 0.9625883102416992, 0.7909668684005737, -0.493863046169281, 1.308210849761963, -1.0621750354766846, -0.15933160483837128, -0.1889672726392746, 0.2886035740375519, -0.4044579267501831, -0.674813985824585, 0.8127991557121277, 0.6833866238594055, -0.020870812237262726], [0.8764826059341431, 0.2650926113128662, 0.3207205533981323, -1.4207872152328491, 1.2250242233276367, 0.4823661148548126, 0.10681665688753128, 0.8240688443183899, -0.30120593309402466, 1.407447338104248, -0.9843517541885376, 0.49551883339881897, 0.047244422137737274, 0.40374404191970825, -0.02367725409567356, 1.1090493202209473, 0.7065118551254272], [-0.8513409495353699, -0.02151672914624214, 0.12283781170845032, -0.5111109018325806, 1.3570647239685059, 0.9184050559997559, -1.3185869455337524, 0.3249519467353821, -0.20346535742282867, 0.814073383808136, -0.5054898262023926, 0.7169134020805359, 0.8626320362091064, 0.20316654443740845, -0.6461663246154785, 0.647321343421936, -1.1529045104980469], [-0.7839793562889099, 0.5560075640678406, 0.20640209317207336, 0.6460625529289246, 0.9439809322357178, 0.36885005235671997, -0.4665273427963257, -0.7007248401641846, -2.021883249282837, 0.3300108313560486, -0.12372709065675735, 0.6634733080863953, -1.157702088356018, -0.16789962351322174, -1.2567830085754395, 1.3087718486785889, -0.16595274209976196], [0.7201419472694397, -1.6111135482788086, 0.15878254175186157, 0.7064921855926514, -0.5700836777687073, 0.5791153907775879, -1.585519552230835, -0.4369169771671295, -1.1255149841308594, 2.4363460540771484, 0.04678643122315407, -0.19487564265727997, -0.42657455801963806, -2.3951573371887207, 0.6102684140205383, -0.5222712159156799, 1.2947640419006348], [0.033010635524988174, 1.0160859823226929, -0.38801366090774536, -0.004922912456095219, -1.2136396169662476, -0.1703813076019287, 1.5916516780853271, 0.396449476480484, -1.3125426769256592, -0.3730928301811218, -0.9774888157844543, 0.19533535838127136, -1.8002163171768188, 0.9517005681991577, 1.5366488695144653, -1.1189075708389282, -0.5756927728652954], [-0.18464292585849762, -1.434503436088562, -0.7433761358261108, 1.5858941078186035, -0.03874558210372925, 0.3702394366264343, 0.3341588079929352, -0.4548965394496918, 1.5490680932998657, -0.40046682953834534, 0.6833741068840027, 0.08703756332397461, 0.5309947729110718, -2.389267921447754, 0.20679064095020294, -1.112310767173767, 0.4161216914653778], [0.1454920768737793, -0.4237057864665985, 0.39660876989364624, -0.39904534816741943, 0.4859113097190857, 1.3018473386764526, 0.02212619222700596, -0.10682793706655502, 0.12863726913928986, -0.12015816569328308, 1.164757251739502, 0.22510021924972534, 2.2394213676452637, -0.343142569065094, 2.0067451000213623, -0.9840325713157654, -0.02972998283803463], [1.2287251949310303, -1.3816968202590942, 0.08044277876615524, 1.5404647588729858, -0.30769822001457214, 1.3642855882644653, -0.21758107841014862, 0.34987208247184753, -0.4068712890148163, -1.1589254140853882, 1.0662057399749756, 1.1387269496917725, 0.44724610447883606, 1.4880716800689697, 3.0275068283081055, 0.04962402582168579, -1.2834761142730713], [-0.2536073923110962, 1.2014240026474, 0.13673751056194305, -2.150233268737793, -0.2977926433086395, -1.3870561122894287, 0.7715627551078796, 0.5448042154312134, 0.7211601734161377, 0.37692081928253174, -1.5524910688400269, 0.8380383849143982, 0.06139187887310982, 0.9006942510604858, 1.3422553539276123, 0.46581676602363586, -1.1514009237289429], [-1.066469669342041, -0.06498628109693527, 0.6418006420135498, -1.5115975141525269, -0.0667356476187706, 2.052849769592285, -0.8998060822486877, -0.41126054525375366, -0.098597951233387, 0.23133276402950287, 0.030230330303311348, -0.6491808295249939, -1.1852656602859497, 1.2850948572158813, -0.401950478553772, 0.4193347990512848, -0.3326226770877838], [0.5463401675224304, 0.7910395264625549, -0.6584847569465637, -2.013176918029785, 0.23764431476593018, -0.10413572192192078, 0.29393628239631653, -0.9940860867500305, 0.5597928166389465, -1.6127088069915771, -0.02595759555697441, 0.027563000097870827, -0.24516674876213074, -1.2659273147583008, -0.37291449308395386, 0.5641443133354187, -1.2116270065307617], [-0.949046790599823, 0.6859771609306335, 1.2595163583755493, 1.5795421600341797, -0.8886463642120361, 0.42036351561546326, 0.9368229508399963, 0.3475383520126343, -0.14527268707752228, 0.04643359035253525, -1.4966093301773071, -0.24807797372341156, 2.1121861934661865, 0.635476291179657, -0.3980599045753479, 0.8347897529602051, -0.642306923866272], [-2.1163995265960693, 0.7479878067970276, 1.2534849643707275, 1.1206966638565063, 1.9952218532562256, 1.531895399093628, -0.8386322259902954, -0.20534740388393402, 1.2462505102157593, -0.8244631886482239, 0.6842082142829895, 0.28567197918891907, -0.6534517407417297, -0.4365626275539398, 0.1171816810965538, -1.1631559133529663, 0.21626347303390503], [-0.986897349357605, -0.6440837979316711, -1.4984276294708252, -0.7017900943756104, -0.3981116712093353, -1.1858586072921753, -1.0836786031723022, 0.9040867686271667, -0.7879494428634644, 2.011803388595581, 1.3752604722976685, 0.1887274533510208, -1.7922842502593994, 0.09359505027532578, 0.32518965005874634, 0.5122250318527222, -0.29540345072746277], [0.2523551285266876, 1.1118286848068237, -0.22126609086990356, 1.078432321548462, 1.0976741313934326, 0.9584602117538452, 1.6683951616287231, 2.523423910140991, 0.7759695053100586, 0.5914008617401123, 0.4893472194671631, 0.3878404200077057, 1.445109248161316, -1.320938229560852, -1.2476202249526978, -0.822433352470398, -0.5221535563468933], [0.7777076959609985, -0.9810944199562073, -0.6498798727989197, -0.3319450318813324, 0.43128108978271484, 1.6764938831329346, 0.07591023296117783, -0.18665720522403717, 0.7791186571121216, -1.379520297050476, -0.3555821478366852, -0.43096739053726196, 2.143683671951294, 0.7009326815605164, 0.7394416928291321, 0.7283555865287781, 0.8866684436798096], [0.9749083518981934, 1.1662508249282837, -0.5197144746780396, -1.3587602376937866, -0.4894440472126007, 0.7130128741264343, 0.23967792093753815, 0.2800978124141693, 0.09924817830324173, -1.8498178720474243, -0.6803491711616516, 0.8245977163314819, -0.8387865424156189, -0.061725664883852005, -1.448626160621643, 0.5045086741447449, 1.0631728172302246], [-0.7856320738792419, 1.7203595638275146, -0.5343848466873169, 0.8775649070739746, 0.09725689888000488, 0.2952880561351776, -0.08992661535739899, -0.3556991517543793, 0.12983866035938263, 1.6008018255233765, 0.10131897032260895, -1.1059963703155518, -0.27048805356025696, 1.0894014835357666, -0.5867142677307129, 1.2068111896514893, -0.05419347435235977], [-1.3601075410842896, -2.010817050933838, -0.9582740664482117, 0.5723189115524292, 0.0084969662129879, -0.0485948845744133, 0.43442636728286743, 1.1540510654449463, -0.2043531984090805, -0.22229400277137756, 0.8507388234138489, 1.3733632564544678, 0.05085606127977371, 1.2871720790863037, 0.6676230430603027, 0.7531945109367371, 1.0597811937332153], [1.204466462135315, 0.951214075088501, -0.6297165751457214, 1.167097806930542, 0.18664120137691498, -1.464525818824768, 1.4118783473968506, 2.502415657043457, -0.11986688524484634, 0.29760274291038513, 0.08144082129001617, -2.0352237224578857, -1.6309555768966675, 1.3390215635299683, 2.0207645893096924, -0.9638552665710449, -0.4094574451446533], [-0.5419541597366333, 1.0997397899627686, 1.1419548988342285, 1.352400779724121, -0.1638084501028061, 2.3338024616241455, 0.42756766080856323, 1.7998079061508179, 1.432705044746399, 0.5029116272926331, -1.1451002359390259, 0.426705926656723, 0.6775310039520264, 0.3794778287410736, 1.1847195625305176, 0.8140655755996704, 0.8673129081726074], [0.8804316520690918, 1.375288724899292, -1.6717697381973267, -0.4374825656414032, -0.8055023550987244, 1.2295899391174316, 1.119126796722412, 1.0354251861572266, 0.04742766544222832, 1.1802198886871338, -0.5603938698768616, 2.038160800933838, 2.0213019847869873, 0.8633546829223633, -1.9899861812591553, -1.4791698455810547, 0.6473028063774109], [0.45124706625938416, 0.024389637634158134, -1.706432580947876, 1.1773165464401245, 1.5496004819869995, -0.5793552398681641, 0.5203883051872253, -1.9440804719924927, -0.21386830508708954, -0.45475873351097107, -0.47165319323539734, 2.191152572631836, 1.5627262592315674, -0.06196664646267891, 0.1967170685529709, 0.0738125815987587, 1.3760448694229126], [-0.7515005469322205, 0.5351701974868774, -0.531531035900116, 0.26506897807121277, -0.3433871567249298, -0.31881120800971985, 0.1421619951725006, 0.6784240007400513, -0.8503954410552979, -0.9745666980743408, 0.20912356674671173, -0.3370221257209778, -0.02194337174296379, -0.7023305892944336, 0.14530999958515167, -0.7388144135475159, 0.27779343724250793], [-0.542942225933075, -1.9173797369003296, 0.4393254220485687, -0.005157783627510071, -0.8234786987304688, -1.8185057640075684, 0.30006909370422363, -1.285884141921997, 0.4760926067829132, -0.09620524942874908, -0.1918913871049881, -0.34650442004203796, 0.4430219829082489, -1.2517645359039307, 1.5178862810134888, -0.5074766278266907, -1.0409749746322632], [-0.391979843378067, -1.3015825748443604, 0.6532809734344482, -1.732859492301941, -0.6135209202766418, -0.35639578104019165, -0.8897563815116882, 1.0421644449234009, 0.19116979837417603, -1.7566620111465454, -0.10570229589939117, 0.1280011534690857, -2.702098846435547, 0.07238586246967316, 0.06469205021858215, -1.7750531435012817, -0.4657951891422272], [0.1403767317533493, 1.099074125289917, 0.45990070700645447, 0.47922027111053467, 0.24156850576400757, -0.6555358171463013, 1.3167341947555542, 0.28176069259643555, -0.04833643138408661, 1.2985986471176147, -1.3281837701797485, -0.40864282846450806, 2.4611570835113525, 1.0202393531799316, -1.8564698696136475, -0.885551929473877, -2.325284481048584], [0.5817314982414246, 0.15578824281692505, 0.25177717208862305, 2.7065908908843994, -0.7627782821655273, -1.193497657775879, 1.7367304563522339, 0.812480628490448, -0.4100160300731659, 0.9647024273872375, 0.17201417684555054, -0.011084704659879208, -1.6070650815963745, -1.8902764320373535, -1.591915249824524, -0.33383414149284363, 0.634213924407959], [1.59445321559906, 0.43395519256591797, 0.2691991627216339, -0.18167263269424438, -0.119842529296875, -0.49008890986442566, -1.4268707036972046, -1.6183301210403442, 0.4723109304904938, -0.4443826675415039, -1.2111186981201172, -0.33438044786453247, 0.01718229055404663, 0.6559213399887085, -0.31410032510757446, 1.497592806816101, 1.173933744430542], [-0.43385958671569824, -0.930622398853302, 0.32446393370628357, -0.7618675231933594, -0.19604209065437317, 1.0311601161956787, 0.001915928558446467, 0.5786939263343811, 2.690354108810425, 0.39745017886161804, -0.008705095387995243, -1.2346594333648682, 0.7962355017662048, 0.2700609862804413, -0.0003403151058591902, -0.5081524848937988, -0.4867473542690277], [2.3469088077545166, -0.7364330887794495, 0.6931886076927185, 0.6532788276672363, 0.050784241408109665, -0.14003345370292664, 0.29729196429252625, -0.7516490817070007, -1.051876425743103, 0.8719809651374817, 0.40113431215286255, 0.6720919609069824, 2.3364248275756836, 0.7304099798202515, 0.7627606391906738, -0.03043774701654911, -0.3320464789867401], [-0.41280698776245117, 0.5308109521865845, -0.3053074777126312, -0.4321657419204712, 1.6356499195098877, -0.4779898226261139, 0.663295567035675, 0.6920003294944763, -0.46111860871315, -0.8382163047790527, -0.3992142677307129, -0.43176573514938354, 0.34792977571487427, 0.1562904268503189, -0.04597692936658859, -0.3521682322025299, 2.0576789379119873], [-1.6956700086593628, 1.4472824335098267, 0.6670355796813965, 0.9480807185173035, -0.12994186580181122, -0.2805577516555786, -0.32300496101379395, 0.2132202833890915, 1.257237434387207, -0.2001323401927948, -0.36900192499160767, 0.48669886589050293, 0.2186124622821808, 0.26790666580200195, -0.7168062329292297, -0.3010583519935608, -0.7017723917961121], [0.18627427518367767, -0.28726717829704285, 0.3144690692424774, 0.843068540096283, -1.8579093217849731, -0.05551096424460411, 0.42582765221595764, -0.07626106590032578, -0.5492991209030151, 0.35635095834732056, -0.2950529158115387, -0.16301004588603973, -0.5740523934364319, 0.2421545684337616, -0.550601601600647, -1.4946753978729248, -2.3028736114501953], [-0.5101667642593384, 1.2242549657821655, 0.6828428506851196, 0.22027619183063507, 2.1345505714416504, -1.0897852182388306, -0.3849150538444519, 0.027966897934675217, 0.6834374666213989, 0.555212676525116, -0.3842729926109314, -0.4805414080619812, 0.416318416595459, -0.8250044584274292, 1.013735055923462, -0.0231203343719244, 0.34762516617774963], [1.3967595100402832, 1.6557432413101196, -1.1594486236572266, 1.2090866565704346, 0.6449771523475647, -1.1341770887374878, 1.3084683418273926, -1.3292902708053589, 0.7678371071815491, 1.2577632665634155, -0.42254218459129333, 0.47053003311157227, 0.6631182432174683, -1.4881975650787354, -0.5265067219734192, -1.0901678800582886, -0.4964870810508728], [-0.9938608407974243, -0.6653579473495483, 1.2723348140716553, -0.23714937269687653, 1.4141128063201904, 0.5696180462837219, 1.6344105005264282, 0.33282577991485596, 0.3049679696559906, 0.31830301880836487, 0.16653862595558167, -0.9177130460739136, -0.8888282179832458, 0.6085206270217896, 1.3493311405181885, 1.176615834236145, 0.21659745275974274], [-0.14015363156795502, -0.5347945094108582, -0.899540901184082, 1.8218669891357422, -2.0142979621887207, 0.6118056178092957, -1.1307569742202759, 0.5060058236122131, 1.649936318397522, 0.9176343679428101, 0.5659970641136169, 0.29026004672050476, 0.5893034338951111, 0.28717556595802307, -0.3148640990257263, 0.18040749430656433, 0.7090252637863159], [-0.0895250216126442, -0.8274993896484375, 0.03456823155283928, 2.096527338027954, 0.09005838632583618, 0.1995202749967575, 1.033764123916626, 0.2572293281555176, 0.7270494103431702, -1.7713394165039062, 0.7119308114051819, -1.374714970588684, 1.011540174484253, 0.18180206418037415, 0.6435559391975403, 1.557421326637268, -0.5725532174110413], [-0.9315826296806335, 0.5396720170974731, -0.5715168714523315, 0.9782288074493408, -0.2918470501899719, 0.6417880058288574, -0.6756018400192261, -0.1511395275592804, -0.5314072370529175, 0.217770516872406, -0.5772919058799744, 0.7296708822250366, -0.01644791103899479, -0.38125354051589966, -0.47864219546318054, -0.17909875512123108, 0.9828065633773804], [0.7050886750221252, 0.5428037643432617, 0.15842242538928986, 0.8707789778709412, 0.8096872568130493, -1.0715094804763794, -0.8300430178642273, 0.22836609184741974, 0.6729771494865417, -0.25404250621795654, -0.19436080753803253, -0.39269939064979553, -0.3789414167404175, 0.5975198149681091, -1.2835755348205566, -1.191979169845581, 0.24776040017604828], [-1.4816830158233643, 0.49455901980400085, 0.2653498947620392, -0.6549303531646729, -2.788611888885498, 0.20323370397090912, -0.22656723856925964, 0.6050072312355042, 2.1552345752716064, 0.41723594069480896, 0.3420660197734833, -0.36618107557296753, -0.2762033939361572, -0.2574617564678192, -1.2828835248947144, -1.7315949201583862, -0.04562729597091675], [0.19488084316253662, 0.340373694896698, 0.3168584108352661, -1.7023593187332153, -0.3003217279911041, -1.0207700729370117, -0.7316180467605591, -0.8510162830352783, -0.882296621799469, -0.34581825137138367, 2.0054969787597656, 0.09445300698280334, -0.664886474609375, 0.7004880905151367, 0.912742018699646, -1.4757474660873413, 1.7297801971435547], [-1.0171489715576172, -1.977757215499878, 1.763299584388733, 0.23878614604473114, -0.5521929860115051, -0.5262187123298645, 0.290874719619751, 0.6416831612586975, -0.8635925054550171, -0.6151086091995239, 0.4591594338417053, 0.7785786986351013, -0.1712246686220169, -0.1369573026895523, 1.137945532798767, 0.6270402073860168, 0.5890264511108398], [-0.643764317035675, 1.7820409536361694, -0.7656479477882385, 0.11382922530174255, -1.949499249458313, -2.0830867290496826, 1.797059178352356, 0.23565885424613953, -0.5310856103897095, -0.5195339322090149, -2.0955023765563965, -1.3954041004180908, -0.9944150447845459, 0.7558047771453857, 0.14078710973262787, -2.066253423690796, -0.05860322713851929], [0.09069313108921051, -0.5295219421386719, 0.29805588722229004, -0.8779107332229614, -0.15367381274700165, 1.5386518239974976, -0.35835152864456177, 1.4627206325531006, 0.728864848613739, 1.153425931930542, 1.1107755899429321, -0.13620099425315857, 0.5285736918449402, -0.7651658058166504, -0.6958380341529846, -0.8355370759963989, 0.21148714423179626], [-0.21270249783992767, 1.3813058137893677, -0.20767979323863983, 0.8232649564743042, -0.7288921475410461, 0.24729101359844208, -0.2313380390405655, 0.63690185546875, 0.03171934187412262, -0.8040632605552673, -0.2021494060754776, -0.5974134802818298, 0.298550546169281, -0.2692372798919678, -0.3947414755821228, -0.6820210814476013, -0.9345670342445374], [0.1831565797328949, 0.8898278474807739, 0.7151387929916382, 0.4316049814224243, 0.9429630637168884, 2.293661117553711, 0.8901636004447937, -0.029831385239958763, -0.27266258001327515, 0.25105389952659607, -1.4377617835998535, 0.21318696439266205, -0.21020205318927765, 0.28156226873397827, 1.0395406484603882, 0.73992919921875, -0.28585347533226013], [-0.5164711475372314, -0.41832712292671204, 0.29757359623908997, 0.4765015244483948, 1.6434025764465332, -0.20507487654685974, -0.19738347828388214, 1.1723791360855103, 0.19285760819911957, -0.08023715764284134, -0.8495428562164307, 0.4865860342979431, 0.49538418650627136, -0.5287127494812012, 2.1406142711639404, 0.5746178030967712, 1.1645641326904297], [-1.2437772750854492, 1.8141224384307861, 0.0881667211651802, 1.1404778957366943, 0.39581045508384705, -0.26111218333244324, -0.8640393614768982, -1.828110694885254, -0.22044356167316437, -1.2054575681686401, 0.28164491057395935, 0.4374985694885254, -1.8234425783157349, -1.09786057472229, 1.9073805809020996, 1.4059581756591797, 2.0125908851623535], [-0.5507574677467346, -1.843181848526001, -0.6137323975563049, -0.15412874519824982, -0.21169023215770721, -0.7519696950912476, -1.304925560951233, 0.17164237797260284, 0.03825443238019943, -0.9224265813827515, 2.226346731185913, -1.1351430416107178, 0.12906430661678314, 0.2735024392604828, 0.32111746072769165, -1.22144615650177, 0.42723390460014343], [-1.0462926626205444, 0.5867889523506165, 0.3252420425415039, -0.8562178611755371, -1.8525875806808472, -0.43425020575523376, 2.04995059967041, 0.4593204855918884, 1.5516068935394287, 0.9234684705734253, 0.1483750343322754, -0.900314450263977, -1.1053788661956787, -1.6270089149475098, 0.47941139340400696, 0.4119530916213989, 1.3124208450317383], [-1.9729822874069214, 0.4314940273761749, 1.6468015909194946, 0.8460479378700256, -0.7165630459785461, 0.8610206842422485, 0.5150424838066101, -0.42698854207992554, -0.7243760824203491, 0.27632999420166016, 0.6436395049095154, 0.6925885081291199, 0.15755382180213928, 0.34551185369491577, 0.30415982007980347, 0.10312074422836304, 2.042776584625244], [-0.5512282848358154, -0.569062352180481, 1.084101915359497, 0.7889952659606934, 1.0594205856323242, -0.2901172637939453, -2.4627163410186768, 1.709102749824524, -0.20557145774364471, -1.1195217370986938, -0.7415831685066223, 0.5573796033859253, 0.5111591219902039, -1.4655895233154297, 0.3380813002586365, 0.6983447670936584, -0.662749707698822], [-0.905236005783081, -0.5259507298469543, 1.225054144859314, -0.8992722630500793, -0.7485333681106567, 0.33737555146217346, 0.2782817780971527, -0.9681156873703003, -0.4121813476085663, -0.13478313386440277, 0.1107986718416214, -0.10688484460115433, 1.4841340780258179, -1.1369447708129883, 0.17019660770893097, 0.4440194368362427, 0.9595695734024048], [-0.07131387293338776, 0.8451555967330933, -1.5459985733032227, 0.3296169936656952, 0.6320328712463379, -0.003272076603025198, -0.8730330467224121, -0.6839044690132141, -0.0008448731387034059, 0.4978826344013214, -0.44361403584480286, 1.4465786218643188, -0.7714539170265198, 2.1992595195770264, -0.8196578025817871, 0.19878947734832764, 0.6548214554786682], [1.1170234680175781, 1.7826645374298096, -0.09438642859458923, -1.9576473236083984, 1.3953098058700562, 1.6227957010269165, 0.5821195244789124, 1.1997671127319336, -2.037172794342041, 0.49541109800338745, 0.21996167302131653, -0.3828543424606323, -0.7433951497077942, -1.3713648319244385, -0.6281115412712097, 0.19946326315402985, -0.7357767820358276], [0.29634976387023926, 0.7361523509025574, -1.8506088256835938, 0.31485873460769653, 1.0633639097213745, 0.7872433662414551, -0.07388083636760712, 1.9880791902542114, -1.0375651121139526, -0.8830157518386841, 0.6240788698196411, 1.6019418239593506, -1.219085693359375, -0.4079408049583435, -1.0287224054336548, -0.5889697670936584, -0.1957092583179474], [1.2543528079986572, -1.053411602973938, 0.35389018058776855, 0.4619433283805847, 0.3156658709049225, -1.7953640222549438, 0.22877000272274017, 1.6071200370788574, 2.255335807800293, 0.8916279673576355, -0.6799387335777283, -0.8088698387145996, -0.03876417875289917, -0.6122750043869019, -0.5167332291603088, 0.21774503588676453, -0.5239843726158142], [0.32556551694869995, -0.07389199733734131, 0.4298095405101776, 0.3892938494682312, 0.6182894706726074, -0.11623425036668777, -1.6107332706451416, 0.7250075936317444, 0.11410681903362274, -2.095181465148926, 0.4697035551071167, 2.396113634109497, 1.0773199796676636, 0.03245677053928375, 2.537107229232788, 0.1528945118188858, 1.2647515535354614], [1.1028246879577637, -0.05347323417663574, -0.1721159666776657, -1.3236491680145264, -0.5753085613250732, 0.9483177065849304, 0.6040549278259277, -0.6850283145904541, -0.930450975894928, 0.5177515745162964, 1.0181571245193481, 0.01761924847960472, 0.32354700565338135, 0.46948525309562683, -0.1700836718082428, 1.3151048421859741, 0.21470828354358673], [-0.47393083572387695, 1.1571829319000244, 1.5668426752090454, 1.004539132118225, -0.6014876365661621, 0.5936142206192017, -0.369860976934433, 1.608929991722107, 1.4106217622756958, -0.5928908586502075, -0.9557691216468811, -1.1482573747634888, 0.3872533440589905, -0.15718276798725128, -1.8367921113967896, 1.2251050472259521, 1.600240707397461], [1.0222022533416748, 1.0266526937484741, -1.2386482954025269, -1.4702413082122803, -0.6041057705879211, -1.0586531162261963, -1.192299723625183, 1.5024160146713257, -1.337712049484253, 0.34556981921195984, 0.7822837829589844, 0.8008036017417908, -0.06258686631917953, -0.9683260321617126, 1.1129496097564697, 0.23652179539203644, 2.0987679958343506], [-0.5807963609695435, -0.666128933429718, 1.9399471282958984, -0.8668151497840881, 3.040480613708496, 0.18883997201919556, 1.8864127397537231, 0.15142662823200226, 0.35090121626853943, -1.154781699180603, -0.011530446819961071, 1.837212085723877, 0.458775132894516, 1.0691708326339722, -0.9425229430198669, -0.41303932666778564, -0.016906406730413437], [-1.9566675424575806, -1.2567203044891357, -0.1338295042514801, 1.3357787132263184, -1.1618775129318237, 2.003946304321289, -1.3574963808059692, 0.7500688433647156, 0.7231895923614502, 2.186894178390503, 0.6761463284492493, -0.846337616443634, -0.32314741611480713, 1.1786926984786987, 1.513175368309021, 0.412767231464386, 0.7980339527130127], [1.3377606868743896, -0.5425295829772949, -1.155154824256897, -0.1772845983505249, 0.2316199690103531, 1.2462568283081055, 0.03312284126877785, -0.05452263355255127, 0.33429816365242004, -0.8643086552619934, 1.0594182014465332, -0.6862747669219971, -0.253682017326355, 0.4664614200592041, 0.9153671860694885, 0.6407275795936584, -0.5913834571838379], [-0.45464417338371277, -1.2120349407196045, 0.7421295046806335, -0.7301856875419617, -0.2593599557876587, 1.3723485469818115, -0.3396911323070526, 0.8142498731613159, -1.0807135105133057, 0.5240442156791687, -1.0495119094848633, -0.47783124446868896, -0.14677460491657257, 0.8844351768493652, -0.606178343296051, 0.8753275275230408, 0.33363157510757446], [1.105688214302063, 0.050317537039518356, 0.2677140235900879, -1.274848222732544, 2.5298917293548584, -1.0020123720169067, -0.29371175169944763, -1.6560553312301636, -0.07307443022727966, 1.0922296047210693, -0.758152961730957, 0.812182605266571, -1.7150071859359741, 0.39097464084625244, 0.13017244637012482, -1.1484793424606323, 1.0618014335632324], [-0.14025293290615082, -0.17703518271446228, -0.21484266221523285, 0.35867539048194885, 0.6450474858283997, -0.42854997515678406, -0.10963907837867737, -2.1533498764038086, -0.7421676516532898, 1.0044339895248413, -0.5225107669830322, 0.42376449704170227, 1.9802958965301514, -0.15961799025535583, 1.233651876449585, 1.0926475524902344, -1.0820584297180176], [1.0127671957015991, 0.7256875038146973, -1.3888185024261475, 1.4229145050048828, 0.8976269364356995, -0.4910361170768738, -1.0101808309555054, 0.165307879447937, 1.2100099325180054, -0.6620984673500061, -0.6857919692993164, 0.7180019617080688, 1.35562002658844, -0.42475008964538574, 0.355506956577301, -0.0726924017071724, -1.3155497312545776], [1.1690759658813477, 2.7456960678100586, 1.6049070358276367, 0.37245848774909973, 1.216842770576477, 0.9903146028518677, -2.6294803619384766, 0.8189545273780823, -1.2121931314468384, 0.9380382895469666, 0.1952623724937439, -1.344267725944519, -0.23841853439807892, 0.8639551997184753, 0.659396231174469, 1.6596282720565796, -1.1730681657791138], [0.4613382816314697, 0.33111050724983215, -0.07958140224218369, -0.5652374029159546, 0.8859626054763794, -0.7845736742019653, -0.033358924090862274, 1.8521755933761597, 0.4341490864753723, 0.0966106429696083, 0.5825552344322205, -0.741016149520874, -1.3033088445663452, 1.0559008121490479, -0.5982951521873474, 1.465507984161377, -0.4297563135623932], [-1.098397970199585, -1.5127849578857422, -0.7034952044487, 0.9048494100570679, 1.5710536241531372, 1.44369375705719, 1.081445336341858, 0.9465087652206421, -1.1362322568893433, 1.3554766178131104, -0.3318372070789337, 0.3514786660671234, -0.30123481154441833, 0.3745264708995819, -0.5842134952545166, 2.1530632972717285, 1.6319046020507812], [0.06171916797757149, -0.17739903926849365, 0.449668824672699, 1.4440237283706665, -0.4437597692012787, -0.24538487195968628, -0.25422215461730957, 1.064798355102539, -0.5868303775787354, 0.09181204438209534, -1.2295050621032715, -1.615909457206726, -1.2762924432754517, -0.22770723700523376, 1.1765753030776978, -1.8682760000228882, 0.10817965865135193], [0.48658719658851624, 0.8648423552513123, 0.46917903423309326, 0.11174347996711731, -0.2573681175708771, -1.7555615901947021, -1.4582884311676025, -0.8993579745292664, -0.2644668519496918, 1.7867196798324585, -0.8558026552200317, -1.1161926984786987, 1.5460002422332764, 0.5135948657989502, 0.27221065759658813, -1.591779112815857, -0.43381431698799133], [-0.8617145419120789, 2.617199659347534, 0.7035320997238159, -0.31058451533317566, -0.42431673407554626, 0.7943089604377747, 0.6489332318305969, 0.5940189361572266, 0.24553613364696503, -0.3032382130622864, 0.07238408178091049, 0.4815416932106018, -0.304108202457428, 0.37077608704566956, -1.7016617059707642, -0.1711454689502716, 0.09253020584583282], [0.940236508846283, 0.17241162061691284, 1.4676483869552612, 0.6449617743492126, -0.9373104572296143, 0.19489005208015442, -0.0777134820818901, 0.7107948660850525, 0.31524378061294556, -0.07313964515924454, -0.14757777750492096, 0.421181857585907, -0.8913497924804688, 0.11335594952106476, -1.1819558143615723, 0.6153092384338379, 0.5728274583816528], [-2.213608503341675, -0.1861814558506012, -0.9571762084960938, 0.6862595081329346, 0.10437718778848648, -0.7798830270767212, 0.6858406662940979, 2.4814999103546143, 0.014994840137660503, -0.7131143808364868, -0.4624600112438202, 0.1260756552219391, -0.17583085596561432, 1.0576953887939453, -0.9395877122879028, -0.6175691485404968, 1.0382068157196045], [0.9605618119239807, 0.36453548073768616, 0.8453538417816162, 0.14416718482971191, 1.9345338344573975, 0.48590174317359924, -1.5744987726211548, -1.143144965171814, -0.7244779467582703, -1.4129835367202759, 0.5108121633529663, -0.6306278109550476, 0.44883257150650024, 0.3779096007347107, 0.4631783962249756, 0.41480642557144165, 0.37149152159690857], [0.8100362420082092, -0.33197221159935, -0.395580917596817, 1.2872990369796753, -1.1153701543807983, -1.3865634202957153, 0.29017338156700134, 0.4526791274547577, -0.6987181305885315, -0.3118400573730469, -0.6042301654815674, -0.3046177923679352, -0.3528910279273987, 2.0964930057525635, -1.3277887105941772, -0.979315459728241, -0.47393059730529785], [1.118090033531189, 1.3454827070236206, 0.4027988016605377, -0.842279314994812, 0.46044278144836426, 0.1211206316947937, 2.2137625217437744, -0.7382447123527527, 1.425728678703308, 1.002333402633667, -0.5846402049064636, -1.4266284704208374, -0.9741612672805786, -0.001197352190501988, 0.904807448387146, -2.0229532718658447, 0.5155103206634521], [-1.2669352293014526, 1.6751112937927246, 0.23448313772678375, 1.8391215801239014, -0.8796852231025696, -0.02167593687772751, 0.3288589417934418, 1.0161131620407104, 1.0098410844802856, 0.3130175769329071, 2.2666749954223633, 0.5780667066574097, 0.3551855981349945, -1.8159574270248413, 1.4243693351745605, 0.10621096938848495, -0.2486708015203476], [1.0619148015975952, 2.22857928276062, 0.6008549928665161, 1.0012164115905762, -1.1769052743911743, -0.45935919880867004, 0.256783664226532, -1.7681595087051392, -0.7891525626182556, 0.9981701374053955, 1.500982403755188, -0.5593706369400024, 0.5658050179481506, -0.6681643724441528, -0.5452694296836853, -0.37897178530693054, -0.39480850100517273], [0.9773188829421997, -1.0997201204299927, -1.031385064125061, 0.8435896635055542, 1.8592748641967773, 1.4954588413238525, -0.24250583350658417, 0.17734217643737793, -1.6019196510314941, -0.6852862238883972, 0.9396193027496338, 0.4558408856391907, 0.5763710737228394, -0.8280448317527771, -1.2072169780731201, -1.0451399087905884, 1.295621395111084], [-1.3125773668289185, 0.35381069779396057, 0.5024172067642212, 1.6380228996276855, -1.0785057544708252, 0.21689046919345856, -0.7514190673828125, -1.6525096893310547, -0.02868765965104103, -0.28572413325309753, -0.26833105087280273, 0.8153195381164551, -0.13820599019527435, -0.06272970885038376, 0.8743403553962708, 0.5288736820220947, -0.2693679630756378], [-0.30154573917388916, 0.6909939050674438, -3.7296955585479736, 0.6003957390785217, 0.8292232155799866, 0.040846388787031174, -1.130811333656311, 0.5189069509506226, 0.9658417701721191, 0.06877094507217407, 0.06782431900501251, 1.1432907581329346, -0.9137881398200989, 0.43332549929618835, 0.8742944598197937, 0.07216577231884003, 1.9145874977111816], [0.5866343379020691, -0.16007137298583984, 1.4012432098388672, -0.9340976476669312, 3.0783236026763916, -0.30185967683792114, 0.4796576499938965, -0.5189427733421326, -0.9104808568954468, -0.41501009464263916, 0.3671974539756775, 0.026453977450728416, -1.3012988567352295, -0.7714881300926208, -0.13477899134159088, 2.096209764480591, -0.7417647838592529], [-0.6543357968330383, 0.47034040093421936, 0.46542707085609436, -0.3127826452255249, -0.6987971663475037, 1.0930004119873047, 0.1788931041955948, -1.2828930616378784, -0.7438096404075623, 1.5438202619552612, -0.7007543444633484, 0.20055778324604034, 0.7268693447113037, -1.0353443622589111, -0.12570194900035858, 0.6556856036186218, -0.5725933313369751], [-0.23103156685829163, 0.7721186280250549, 0.5680916905403137, -0.22774076461791992, -0.22476112842559814, 1.162413239479065, 0.8302067518234253, -1.024040937423706, 0.538073718547821, -1.5929213762283325, -0.7450549006462097, 0.06928066164255142, 1.0702093839645386, 0.5657154321670532, -0.7419891357421875, 0.8500198125839233, -0.2489093691110611], [0.8514958620071411, 0.7432021498680115, -0.39660510420799255, 1.0675185918807983, -0.15957197546958923, -0.1624210923910141, 0.2650079131126404, 0.37471893429756165, -1.581096887588501, 1.1671781539916992, 1.1524620056152344, -1.1032898426055908, 0.8614975810050964, 1.135425090789795, 1.3324521780014038, 0.8263995051383972, 0.6223184466362], [0.502748429775238, 0.2311680018901825, 0.5877718925476074, 0.12223690003156662, 0.9750763773918152, 0.8837365508079529, -0.5703580975532532, -1.8306502103805542, -2.292184352874756, -0.22955003380775452, -0.3820459246635437, 0.6179656386375427, -0.9021217226982117, -0.2586863040924072, -1.4423378705978394, 0.6533476710319519, 1.2233000993728638], [-2.089756965637207, 0.10288097709417343, -0.9230787754058838, -1.8836525678634644, 0.04469601437449455, 1.7784239053726196, 0.2438538670539856, -1.2338011264801025, -0.4478968679904938, 1.0062974691390991, 0.15531408786773682, 2.4656665325164795, -1.3388617038726807, -0.8527256846427917, -0.09169524908065796, 0.10279382020235062, 0.6579526662826538], [0.6873932480812073, 0.8664583563804626, -0.9013482928276062, 0.6810412406921387, -0.7772794961929321, 0.780442476272583, 0.40801841020584106, -1.4630591869354248, 2.051933765411377, 0.22137950360774994, -1.0362669229507446, -0.3563891351222992, -1.1504127979278564, -0.5982189774513245, 2.27416729927063, 1.0509376525878906, -1.0498751401901245], [1.927725076675415, 0.41854462027549744, -0.5223963260650635, -1.3185691833496094, -0.6257713437080383, 0.37590292096138, -0.6075253486633301, -0.5801843404769897, -2.393846035003662, 0.8911828398704529, 0.8901554346084595, -0.9420675039291382, -0.5969114303588867, 0.5189475417137146, -0.3405466079711914, -1.5853850841522217, -0.49174585938453674], [0.18882602453231812, -0.523639976978302, -1.2032480239868164, -0.9526892304420471, -0.6651316285133362, -2.2504539489746094, -1.1504809856414795, -0.17731039226055145, -1.1880964040756226, -0.26859134435653687, -0.15112243592739105, 0.4518962800502777, 1.9629143476486206, -0.45176833868026733, -0.4606161415576935, 0.10020019859075546, -1.0145320892333984], [0.3030151426792145, 0.05073026940226555, 0.10251011699438095, -0.591698944568634, -0.27927157282829285, 0.8380492329597473, 2.121278762817383, 1.514263391494751, 0.9463887214660645, 0.8374274969100952, -0.2927873730659485, -1.0271079540252686, -0.45657238364219666, 0.24251699447631836, -0.20736278593540192, 0.39834389090538025, -1.0071346759796143], [1.0548439025878906, -1.1384820938110352, 0.5600957274436951, -0.3488968014717102, 0.4209648072719574, 1.5976479053497314, 1.1683452129364014, -1.1192454099655151, 1.3693856000900269, -0.07759474962949753, 0.3382561504840851, 0.07426832616329193, -0.41575804352760315, -1.0490738153457642, -0.7237827181816101, -0.8852047324180603, 0.5495412349700928], [-0.9275503158569336, 1.6134713888168335, -1.2811886072158813, 0.9413644075393677, -0.11640674620866776, -0.8007593750953674, -1.4336886405944824, 0.6965110898017883, 0.015392640605568886, -0.11335232853889465, -0.03244917839765549, -1.2449097633361816, -0.9924609065055847, -0.2804414629936218, 1.6891658306121826, -0.07227461785078049, -0.1630120426416397], [-2.260831356048584, -0.5387457609176636, 0.5511474609375, 0.493307501077652, -0.5549697279930115, 0.7283914089202881, 0.4834546148777008, -1.68287193775177, -1.241501808166504, 1.2186217308044434, -1.2854082584381104, -0.9709905385971069, 1.3018393516540527, -0.05599908530712128, 0.7071231603622437, 0.5541031360626221, -0.15530064702033997], [-0.9148691892623901, -0.3758004605770111, 1.2161437273025513, 1.1066625118255615, 1.871978998184204, -1.7077860832214355, -1.9858802556991577, 0.6663910746574402, 1.0293585062026978, -0.03204755857586861, 0.09533502906560898, 1.6373064517974854, 0.3999212980270386, 0.17663222551345825, 0.847979724407196, -0.9372692704200745, 0.5680275559425354], [-1.3534963130950928, -0.4302397668361664, -0.27748411893844604, -0.9440364241600037, 0.1685037612915039, 0.4695756733417511, 0.2519094944000244, -0.14130240678787231, 0.8610790371894836, -0.8416025042533875, 2.5986533164978027, 0.3720056414604187, 0.31396418809890747, 1.112295389175415, 0.40697774291038513, -1.7819565534591675, 1.4925706386566162], [1.096097707748413, 1.3662549257278442, -1.7187696695327759, -1.8583440780639648, 0.8931719064712524, 0.7461578845977783, -0.39064908027648926, -0.052038632333278656, -0.0729442909359932, -0.15443085134029388, -0.9562792181968689, -0.9975992441177368, -1.6508514881134033, -0.9025925397872925, 1.5454349517822266, 1.6597131490707397, -1.9863837957382202], [1.0224745273590088, 0.35158854722976685, -1.1551117897033691, -2.0604307651519775, -1.414776086807251, -0.9957517385482788, 0.9728268384933472, 1.8223304748535156, 0.06472819298505783, -0.03796575218439102, -0.9529412388801575, -0.7915096879005432, -0.20106570422649384, 1.0941476821899414, -0.08570357412099838, -0.685116171836853, -1.2641931772232056], [-0.6494980454444885, 0.6603621244430542, -0.22938963770866394, -1.206120491027832, 0.3269585072994232, -0.4794941544532776, 0.8471291661262512, -1.6843105554580688, 0.46216681599617004, 0.21668973565101624, -0.1655770242214203, -0.484796404838562, -1.619594931602478, -0.6514215469360352, -0.30511748790740967, 0.36623337864875793, 0.061456430703401566], [-0.16068843007087708, 1.0605385303497314, 0.4425188899040222, -0.11598461866378784, 1.0315362215042114, -0.17751359939575195, 0.43972209095954895, -0.20784428715705872, -0.14675255119800568, -0.15960688889026642, 1.49038827419281, 1.1664626598358154, 1.162468433380127, -1.200063943862915, 1.161219835281372, 0.8519312739372253, 0.7289212942123413], [0.6739422082901001, -0.5614621043205261, 0.6906052231788635, 1.3155624866485596, -1.2159576416015625, 0.7784050107002258, 0.14950108528137207, 0.042340975254774094, 0.7007485032081604, -0.5868732333183289, 1.8055405616760254, -1.8023483753204346, -0.11450109630823135, -0.2685457468032837, 0.38375887274742126, -2.3433077335357666, 0.9273796081542969], [-1.8641809225082397, 0.13237543404102325, -0.46227240562438965, 1.153652310371399, 1.7914164066314697, -0.31163290143013, -0.3532741069793701, 0.2413591891527176, 1.1747570037841797, -1.2955940961837769, 0.8116562962532043, 0.7699581980705261, -1.5388858318328857, 0.5192822217941284, -0.3010033965110779, 0.6823861598968506, 0.506035327911377], [-0.8141362071037292, 0.6159859299659729, -0.5420634150505066, -0.17474021017551422, 0.10380759835243225, -0.8197165727615356, -0.626477837562561, 0.7006429433822632, 0.229127436876297, 0.5792875289916992, -0.34751248359680176, 0.22738510370254517, 0.9740741848945618, 0.7978237271308899, -0.11072363704442978, -0.14258034527301788, -0.6920378804206848], [-0.7491122484207153, 1.5186538696289062, -1.3543694019317627, 0.7296481132507324, 1.6891165971755981, 0.021020498126745224, 1.2405903339385986, -0.08669822663068771, -0.680742084980011, -0.9699210524559021, 1.040643334388733, -0.17269523441791534, 1.2093013525009155, 0.5317724943161011, 1.1932742595672607, -0.32144516706466675, 0.47758328914642334], [1.717093586921692, -3.05568528175354, 1.337719202041626, -1.08937406539917, -1.264370083808899, -0.8438066840171814, 0.3314899802207947, 1.0979955196380615, -2.132922649383545, -1.3760911226272583, -1.4975788593292236, 0.37524667382240295, 0.14337120950222015, -0.3230045437812805, 0.03694825991988182, -0.9583279490470886, -0.5591036677360535], [0.5000885725021362, 0.20245715975761414, 1.5644656419754028, -0.4633623957633972, 1.6269710063934326, 0.1944320797920227, 0.8236457705497742, -0.9278030395507812, 1.9790984392166138, -0.18857960402965546, 1.398264765739441, -0.4439080059528351, 0.26437801122665405, -0.12373509258031845, -1.0329822301864624, -2.155906915664673, 1.0488539934158325], [-1.6019152402877808, 2.2087135314941406, 0.31420400738716125, -0.5460126996040344, 0.9483346343040466, -1.2492409944534302, -1.961949110031128, 0.38672295212745667, 0.6428678035736084, -0.22478087246418, -0.005184292793273926, -2.5241832733154297, -1.1146801710128784, -1.1258920431137085, -0.446445494890213, -0.9747201800346375, -0.8194838166236877], [-0.47814446687698364, -0.9076908826828003, -1.0523017644882202, -0.5602514147758484, -1.2686103582382202, -0.5874989628791809, -1.0833775997161865, -0.18275511264801025, 0.7331964373588562, 0.07061070203781128, -0.6489139795303345, 0.6718785166740417, -0.47003281116485596, 1.1324458122253418, 0.5644416809082031, -1.213915228843689, 1.2521981000900269], [0.6172464489936829, -2.0791351795196533, 1.4369163513183594, 1.4262537956237793, 1.1533300876617432, 1.0776335000991821, 0.31020069122314453, 1.4071954488754272, -0.12993931770324707, -0.9478013515472412, -0.2889930009841919, -0.6864099502563477, -1.1794019937515259, -0.7593464255332947, -0.24921159446239471, 1.4018962383270264, 0.20471103489398956], [1.0305503606796265, -0.1858326494693756, 1.1502172946929932, 0.37650808691978455, -0.516395092010498, 1.4903755187988281, -0.6057898998260498, 1.6960256099700928, -1.8952301740646362, 1.3637382984161377, 0.5152254104614258, -0.48522698879241943, 0.19974148273468018, 0.09150834381580353, 0.31940484046936035, -0.36659011244773865, -0.4918915331363678], [-0.103984534740448, 0.16059580445289612, -2.0496068000793457, 0.8678820133209229, 0.8400020599365234, 0.3849300444126129, 0.3545463979244232, -0.6808735728263855, 0.5197211503982544, -1.4125547409057617, 1.0782556533813477, 1.0320848226547241, 0.7272014617919922, 0.5636423826217651, -0.02771010249853134, -0.22489140927791595, 0.18781588971614838], [-0.8751678466796875, -0.020825611427426338, -0.11085415631532669, -2.1035735607147217, -1.408810019493103, 1.4342105388641357, -0.46624788641929626, -1.2767302989959717, 0.9980688691139221, 1.7416459321975708, -0.37892812490463257, 1.0784841775894165, -0.6314745545387268, -0.13905073702335358, -0.7601898908615112, -0.11913896352052689, 0.3919927775859833], [-0.9110006093978882, -1.5370880365371704, 1.5595303773880005, 0.012767402455210686, 0.18386413156986237, -0.5046348571777344, 1.1405091285705566, -0.5225862264633179, -0.26656925678253174, -0.6322829723358154, -0.010159691795706749, 2.477088451385498, -1.073439598083496, 0.04647938534617424, 0.8506417274475098, 0.5128677487373352, -1.4426560401916504], [0.22321298718452454, 0.005898685194551945, 0.5643659234046936, -0.6774669885635376, 0.4352596700191498, 0.21939219534397125, -0.8994792103767395, -1.0383943319320679, -0.330548495054245, -0.34797272086143494, 0.6442669630050659, 0.7073636651039124, 0.17023080587387085, -0.6053876876831055, -0.9596787095069885, 0.3815987706184387, -1.0826642513275146], [2.683140277862549, -1.0363483428955078, 0.6189817190170288, -0.4791173040866852, 0.33156904578208923, 0.41909804940223694, 1.1736071109771729, 1.8905099630355835, 0.43071913719177246, 1.640486478805542, 0.4891553223133087, -0.8164798021316528, -0.06574035435914993, -0.5439565181732178, 0.435616672039032, 1.3161747455596924, 1.6584796905517578], [-0.5683196187019348, 0.5670069456100464, -0.8348188996315002, -0.272281676530838, -1.4280694723129272, 2.1838858127593994, 0.6437456607818604, -0.23837459087371826, -1.3962492942810059, -0.27967023849487305, 0.8824485540390015, 0.012769696302711964, 0.37086087465286255, -0.25269776582717896, -0.6709635257720947, 0.9470911026000977, -0.6990619897842407], [0.5582051277160645, -0.7400530576705933, 0.21192583441734314, -1.2456691265106201, -0.594291090965271, -0.5903400778770447, -0.2899460196495056, 1.1340968608856201, -1.0872963666915894, -0.07714743912220001, -0.45541346073150635, -1.221627116203308, 0.09276647865772247, -0.2841593623161316, -0.7915440797805786, -2.660949230194092, 0.02581172063946724], [-0.1101456731557846, 1.3124617338180542, -0.7956870198249817, -0.7068787813186646, 1.2066118717193604, -0.03940245136618614, 0.7905467748641968, -1.5523115396499634, 0.24175578355789185, -0.01290070079267025, 0.39462974667549133, 0.7777018547058105, 1.3557095527648926, -1.0691076517105103, -2.475003480911255, -0.7677673697471619, -0.2728061079978943], [0.43924596905708313, -0.3531278073787689, -0.5093868970870972, -0.9408491849899292, -0.044456396251916885, -0.4501885175704956, 1.5720888376235962, -1.2707595825195312, 1.4711323976516724, 1.3391385078430176, 0.3171521723270416, -1.5634653568267822, -0.1560860574245453, 0.25482940673828125, -2.2211508750915527, -0.11568881571292877, -1.4976081848144531], [1.5740306377410889, 0.49596869945526123, 0.19160878658294678, -0.5645768046379089, 0.23976197838783264, 0.9358744621276855, -0.7021525502204895, -0.3356291949748993, 0.525307297706604, 0.15536358952522278, 0.5261089205741882, 0.04091918095946312, -0.27472755312919617, 0.3739643096923828, -1.0328205823898315, 0.05437473952770233, 0.9119638204574585], [-0.21465671062469482, 1.0844987630844116, 0.06872359663248062, 1.0216532945632935, 1.645278811454773, -1.88649320602417, 0.4641290307044983, -0.24454127252101898, 0.19901683926582336, 0.849348247051239, 1.4376213550567627, -0.5849734544754028, 0.63755863904953, 2.031158447265625, 2.094329833984375, -2.4550914764404297, 0.1832963228225708], [0.34393808245658875, -0.7111501097679138, 0.30345168709754944, -0.21147918701171875, 0.6797009706497192, -0.7579903602600098, 1.298742651939392, -0.7092574238777161, 0.3190228343009949, 0.15903565287590027, 0.6527714729309082, -1.0841944217681885, 0.2894551753997803, 0.6408471465110779, 1.9501711130142212, 0.500342607498169, -0.8728480339050293], [0.18880972266197205, 0.7259098887443542, -0.11858241260051727, -0.8416417241096497, -1.7594661712646484, 0.17919224500656128, 0.059265874326229095, -0.6895706653594971, -1.4626502990722656, -1.216060757637024, -0.8267825245857239, -0.6758514642715454, -0.2017093002796173, -0.7819285988807678, -1.0054585933685303, -0.618116021156311, 0.3533308207988739], [0.07318616658449173, -1.1296759843826294, 0.6230104565620422, 0.10323401540517807, 0.6724563241004944, 1.6120657920837402, -0.7777012586593628, -1.1995373964309692, 0.28264033794403076, 0.6993932127952576, 2.309537649154663, 1.1496249437332153, 0.47711092233657837, -0.6227676272392273, -0.02100820094347, 0.8313933610916138, -2.089479446411133], [0.4485735595226288, -1.5111608505249023, 2.8531010150909424, 1.267510175704956, 2.2563788890838623, 0.6656259894371033, 0.12905991077423096, -1.444680094718933, -0.9997857809066772, -1.2694096565246582, -0.6535246968269348, 0.22368232905864716, -1.553460955619812, -1.9493374824523926, 0.5053113698959351, -0.3018092215061188, -0.730000913143158], [0.10752563923597336, -1.1220645904541016, 1.3514420986175537, 0.09508063644170761, -0.3079848289489746, 0.047730688005685806, 1.845070242881775, -0.6971452236175537, -1.4254205226898193, -0.46673691272735596, -1.776319146156311, 0.9134270548820496, -2.4880928993225098, -0.8094790577888489, 1.7375391721725464, 0.6943899393081665, -2.9173507690429688], [-0.388102263212204, 2.5921480655670166, -0.22242523729801178, -1.038437843322754, -0.862673282623291, -0.6765900254249573, 1.0583434104919434, 0.8715616464614868, 0.0704561397433281, 0.3953527510166168, -1.148727297782898, 0.47315603494644165, -1.7182217836380005, 1.8277137279510498, -0.10604387521743774, -0.4835841953754425, -1.0337214469909668], [-0.25838756561279297, -0.8162648677825928, 0.45062145590782166, 0.5559703707695007, -1.3083926439285278, -0.48447737097740173, 0.6590254306793213, 1.6966203451156616, 0.6280338764190674, -0.9672664999961853, -0.8368618488311768, -0.6403163075447083, 2.6674084663391113, -1.2143967151641846, -0.04785463586449623, 0.517509400844574, -0.41546717286109924], [-0.22953489422798157, 1.9941061735153198, 0.545997142791748, -0.10040263831615448, 1.7211400270462036, -0.9160582423210144, 1.619278073310852, 0.526107132434845, -0.2553859055042267, 1.1557456254959106, 2.428922414779663, 0.32443737983703613, -0.39323538541793823, 1.5539801120758057, 0.5402079820632935, 1.1864725351333618, 1.008834958076477], [0.5294619798660278, 1.6454973220825195, 0.6533238291740417, -1.7357741594314575, 0.4528644382953644, -0.47227904200553894, 0.879601776599884, -0.401719331741333, -1.9473165273666382, 0.13332411646842957, 0.48939454555511475, 0.5268047451972961, -0.21000850200653076, 1.9396111965179443, -0.6724967956542969, 0.3576671779155731, 0.2521265745162964], [-0.10293898731470108, -0.07978543639183044, 0.34770864248275757, 0.9907376766204834, 0.14491704106330872, -0.2518513798713684, -1.968492865562439, -0.2117239534854889, -0.1680225282907486, -1.3829984664916992, 0.36459577083587646, 1.1408249139785767, -0.471671998500824, 0.2718368172645569, 0.3080962896347046, -1.3734140396118164, 1.0717319250106812], [-0.5271483659744263, -0.9244495630264282, -2.2093493938446045, 0.0895705446600914, -0.47653061151504517, 0.8587490916252136, -0.4054621160030365, 0.4183007776737213, 1.1816011667251587, -1.0367475748062134, -1.5522958040237427, -0.7000216841697693, 1.046817421913147, -0.1295742690563202, 0.58167564868927, 1.0041656494140625, -0.7606070637702942], [0.08851716667413712, 0.6498079299926758, 0.47483083605766296, -0.5682940483093262, 1.4651292562484741, 0.6925514936447144, -2.6223325729370117, 0.7470886707305908, 0.06294221431016922, -0.43106111884117126, 0.6128177642822266, 0.6595820188522339, -1.3522236347198486, -0.9881830811500549, -0.7122028470039368, 1.381968379020691, 0.3148971199989319], [0.15726421773433685, -1.8414219617843628, -0.8276488780975342, 0.9890127778053284, 1.1838569641113281, 0.28863340616226196, -1.3112776279449463, -0.4741824269294739, -0.17907387018203735, -0.5947738289833069, 1.0338658094406128, 0.2474793642759323, -0.6151180267333984, -1.0711673498153687, 0.4104881286621094, 0.31992849707603455, 0.9640825390815735], [-1.3464008569717407, -2.0222063064575195, 1.3897318840026855, -0.7257639765739441, -0.08109939098358154, -0.1052837073802948, -0.7630115151405334, -2.0545568466186523, 0.7920392155647278, 1.200188398361206, 0.4016270041465759, -0.6872930526733398, 0.6506210565567017, -0.4656517803668976, 0.15201370418071747, 0.06937482953071594, 1.014297604560852], [-0.6834575533866882, 2.0375452041625977, -1.2050632238388062, -0.6425871849060059, 0.4538242220878601, -0.2141534388065338, -0.28140199184417725, -1.1110385656356812, -0.07021091133356094, 0.5720473527908325, -0.3455280661582947, -0.7195876240730286, 0.8744989633560181, -2.85968017578125, 0.915442943572998, -0.4056583642959595, -1.0215636491775513], [-2.0714292526245117, 0.3765990436077118, -0.8911980390548706, -0.4310673475265503, -1.078156590461731, -1.2077897787094116, 0.7749556303024292, 2.007840871810913, 0.1280025988817215, 0.6359355449676514, -1.539005994796753, 1.7187597751617432, 1.2761040925979614, 0.20173655450344086, -1.3113874197006226, -1.3692563772201538, -1.9023529291152954], [-1.2803571224212646, -0.21006463468074799, -0.3738206923007965, -1.3257927894592285, 0.8663146495819092, -0.8822863101959229, 1.3208446502685547, 0.8879131078720093, -0.35723966360092163, -0.4022822976112366, -0.24028268456459045, -0.9198054075241089, -1.0031747817993164, -1.1047190427780151, -1.3962173461914062, 0.39830973744392395, 0.008578704670071602], [-1.3739084005355835, 0.4574790298938751, 1.5107769966125488, -0.49247854948043823, 0.6455551385879517, 1.5822312831878662, 0.7706777453422546, 0.46554896235466003, -0.35544484853744507, -0.5413802862167358, 0.6546945571899414, -0.6337438821792603, -0.20191198587417603, -0.6395712494850159, -0.008709809742867947, 1.6273139715194702, -0.6038391590118408], [1.4287134408950806, -0.29474344849586487, 0.5073971748352051, 0.75518798828125, -0.9463734030723572, -0.49748873710632324, -1.372815728187561, 0.7120154500007629, 1.3908919095993042, -1.3072242736816406, -2.001269817352295, -0.8075544238090515, -1.3247274160385132, 0.6746683120727539, 0.22914549708366394, 0.2331041544675827, 1.0060925483703613], [1.052971601486206, 1.037078619003296, 0.9741839170455933, -0.4637235999107361, -1.4549920558929443, -0.7984664440155029, 0.6601214408874512, -1.9105942249298096, -0.10053504258394241, -0.05752595141530037, -0.4189240038394928, -0.35486486554145813, 0.30459296703338623, -0.5183904767036438, 0.49401184916496277, 1.3377685546875, -0.02673480287194252], [-0.7946605086326599, -0.5779018402099609, -2.1161487102508545, 0.99528968334198, 0.41805437207221985, 1.1348272562026978, 0.9461281895637512, 1.2633343935012817, 0.9613827466964722, 0.5794322490692139, -0.22386644780635834, 1.9045956134796143, 1.0486372709274292, -0.9693374633789062, 0.5219610929489136, 0.36953961849212646, 3.105848550796509], [0.44386979937553406, 1.5060659646987915, 1.3153252601623535, -0.2621321976184845, -0.5896568298339844, 0.33108779788017273, -0.6165077686309814, 0.4095611870288849, -0.36595040559768677, -0.04397668316960335, 0.3066047430038452, 1.6885583400726318, 1.0829157829284668, 1.7027696371078491, 1.1089450120925903, -1.8121798038482666, -1.325278639793396], [0.5854886770248413, 1.781399130821228, 0.6137833595275879, 1.7836748361587524, -0.4462563395500183, 1.0428252220153809, -0.6226580739021301, -0.03151584416627884, 1.766281247138977, -1.4086434841156006, 0.0853215679526329, 0.21558062732219696, 1.371200680732727, -0.40787190198898315, -0.0007703453884460032, 0.8771605491638184, 1.8108628988265991], [0.06303613632917404, 0.40988215804100037, -0.037163954228162766, -0.1803678274154663, 0.025204287841916084, -0.5516842603683472, -0.5514237880706787, -0.37475475668907166, 1.6643378734588623, 0.175442174077034, -1.2217345237731934, -0.7767011523246765, -0.5685892701148987, -1.4783034324645996, 0.18799136579036713, 0.2815735638141632, 0.04233601316809654], [1.8255529403686523, -0.07064060121774673, 0.15210115909576416, 0.4438559412956238, 1.9754334688186646, -0.6422028541564941, 1.621274471282959, -0.059542346745729446, -2.4051918983459473, -0.29832330346107483, 0.7041183114051819, 2.709805488586426, 0.37785011529922485, -2.4665000438690186, 1.430295467376709, -0.6798758506774902, 0.26449325680732727], [0.9930926561355591, -0.6365361213684082, -1.0371633768081665, -0.3935086727142334, -0.06131584197282791, -0.6609175801277161, 0.6017017364501953, -1.1367170810699463, 0.7713224291801453, -0.8191209435462952, 1.4029302597045898, -0.261025995016098, -1.012753963470459, 1.2062404155731201, -0.2624324858188629, 1.4469273090362549, 0.5964514017105103], [-1.7108705043792725, 0.4547747075557709, 1.6562297344207764, 0.1969563364982605, -0.5525681376457214, 0.1899813860654831, 0.3120385408401489, -1.4058371782302856, -0.5725783109664917, -0.9000974297523499, 1.8220211267471313, 0.2547173500061035, 0.8507851958274841, 1.3129515647888184, 2.39564847946167, 1.4882357120513916, -0.5594409108161926], [-1.1247494220733643, 0.9417317509651184, 0.06140998378396034, -0.5043370723724365, 2.9266176223754883, 0.10772745311260223, 0.10626208037137985, -0.5144652128219604, -0.5528507232666016, 1.254300594329834, 0.8196772933006287, -0.2201995700597763, 1.6449530124664307, 0.7393063306808472, 0.6559197902679443, -1.5044727325439453, -0.3233424723148346], [-0.08054568618535995, -0.7981322407722473, 0.136640727519989, 0.5319982171058655, 0.33827850222587585, 1.1678364276885986, -2.278545618057251, 0.6423695087432861, 1.3982967138290405, 1.4196653366088867, 0.18375812470912933, 0.4621170163154602, 0.4102153480052948, 1.64277184009552, 0.1552659273147583, 1.521385669708252, -0.6181958317756653], [2.2081522941589355, 0.6565797924995422, -0.45063579082489014, -0.3017530143260956, -2.269170045852661, 0.4800770878791809, -0.05723840370774269, -0.8439459204673767, -0.05144261196255684, 1.5564892292022705, -0.4020053446292877, 0.4360245168209076, 0.03919830173254013, -0.6575876474380493, -0.7601655125617981, -0.110898457467556, -0.09631681442260742], [0.12000540643930435, 1.7044342756271362, -0.14829044044017792, 0.16340221464633942, 0.865854024887085, 0.3500128984451294, -1.2725017070770264, 0.2604047954082489, -0.7450245022773743, 0.4368385672569275, -1.7875986099243164, 1.595775842666626, 0.40288496017456055, -1.9561179876327515, -0.41926589608192444, -0.9778914451599121, 1.055078148841858], [1.180932879447937, 1.3454465866088867, 1.2286993265151978, 2.0382490158081055, -0.6932576298713684, 0.921323835849762, -1.7024234533309937, 2.206265687942505, -2.259180784225464, 0.4546811282634735, -1.8249645233154297, 0.019358640536665916, 0.14892396330833435, 0.14756925404071808, -0.3340834975242615, 0.0006629059207625687, -0.43367016315460205], [-0.31945404410362244, 0.22979940474033356, -1.5709973573684692, -0.09097308665513992, 1.0550806522369385, -1.4551488161087036, 2.457306146621704, 2.8495872020721436, 0.2360209971666336, 0.4220735728740692, 0.5694208741188049, 0.8251441717147827, -0.10762802511453629, -0.8415510058403015, 0.4720557928085327, -0.2151748538017273, 0.08135098963975906], [1.523477554321289, 0.2021934539079666, -1.0747475624084473, -2.486959457397461, 1.2525039911270142, -0.732989490032196, 0.31279292702674866, 0.6141530275344849, -2.0847339630126953, -0.7647905945777893, -0.011220366694033146, 1.1345207691192627, 0.2297067642211914, 0.7948681116104126, 1.8948348760604858, 0.6100401282310486, -0.4420403838157654], [-0.3079116940498352, 0.06263954192399979, -0.3118569552898407, -0.13683989644050598, 0.8063256740570068, 1.2391963005065918, 0.6196143627166748, -2.357975721359253, 1.7558804750442505, 0.8698327541351318, -0.24119675159454346, 1.4707372188568115, 0.6360738277435303, 0.8984981179237366, 0.22848112881183624, -1.201887607574463, 0.5188608765602112], [-0.22827643156051636, -0.39898407459259033, -1.1858887672424316, -1.1391645669937134, -0.5674466490745544, -0.5938129425048828, -1.7242817878723145, 0.07775569707155228, -0.21608251333236694, -0.5734673142433167, 0.09300044924020767, 0.008778342045843601, -0.06737277656793594, -0.3917173743247986, 0.7799303531646729, 2.007148504257202, -0.0453292615711689], [-1.1930320262908936, -0.8056923151016235, -0.6795560121536255, -0.04109441488981247, 0.3852841854095459, 0.7444408535957336, -0.6978740692138672, -0.10181794315576553, -1.6710964441299438, 0.16259212791919708, 0.7825767397880554, -1.64486563205719, -0.5713693499565125, -0.5438869595527649, -2.253959894180298, -1.041424036026001, 1.2942123413085938], [0.5818946361541748, 0.04037322476506233, 0.04511009901762009, -0.301146924495697, 1.7293188571929932, 0.13982979953289032, -1.1236382722854614, 0.8831896185874939, -0.5692944526672363, -0.5847988128662109, -0.6579229235649109, 0.38133952021598816, -0.16423821449279785, 0.8145388960838318, -0.05003243312239647, -0.10949792712926865, -0.16391898691654205], [0.025061076506972313, 0.44035694003105164, 1.370707631111145, -0.8541332483291626, 1.3496567010879517, 1.1964855194091797, -0.7262521982192993, 0.5418728590011597, -0.5672854781150818, -0.5240281224250793, 0.24152179062366486, -0.4765094220638275, 1.4477393627166748, -0.7675555348396301, -0.8616313338279724, 1.0165207386016846, -0.3531225025653839], [0.9663108587265015, -0.8974186778068542, -1.3241573572158813, 0.05631984770298004, 0.698003888130188, 0.8054980635643005, 0.4583689272403717, -0.5569162964820862, -0.6565325260162354, 0.44520384073257446, -1.253890037536621, 1.2348806858062744, 1.029684066772461, 0.4068264365196228, 1.4199706315994263, 0.08822284638881683, 1.5179028511047363], [0.7103981375694275, 0.023043358698487282, 1.0319324731826782, 1.659487009048462, 0.5027052164077759, 0.0053107161074876785, 1.6269934177398682, 0.9870269894599915, -1.1037026643753052, 1.0984572172164917, 0.12134673446416855, 0.5796120762825012, 0.04351234808564186, 0.8640573024749756, -0.5961989760398865, 0.5928229093551636, -1.5951145887374878], [-0.3172089755535126, -0.4256415367126465, -0.6987098455429077, -1.7790980339050293, -1.3161808252334595, -0.24151258170604706, -1.4147297143936157, -1.2025229930877686, -0.0576774962246418, -0.5891926288604736, 0.7889643311500549, 0.4627835750579834, 0.2165219485759735, 0.343425452709198, -0.07798115164041519, -0.8946738839149475, -0.6757782101631165], [0.31235623359680176, -0.35540419816970825, 0.35506027936935425, -0.6579264998435974, -0.9691953659057617, -0.7284268736839294, -0.7077524065971375, -1.8084222078323364, -1.1393953561782837, -1.7861237525939941, -0.36282676458358765, 1.930539608001709, 0.8614959120750427, -0.20853419601917267, 0.28153231739997864, 2.5517544746398926, 2.293102979660034], [-0.10550274699926376, -1.9442871809005737, -0.7083348631858826, 0.9506680965423584, 1.6279125213623047, 0.5529727935791016, -0.8099014759063721, 0.5042299032211304, 1.0788803100585938, -0.36825305223464966, 0.33966711163520813, 0.8690955638885498, -0.6632303595542908, -0.4700012505054474, 1.3274625539779663, 0.27467837929725647, 2.4668898582458496], [0.08761520683765411, -0.6523658633232117, 0.4095860421657562, 2.2700839042663574, -0.4549062252044678, 1.703015923500061, -1.1838090419769287, 0.5181727409362793, -0.5235045552253723, 0.08563265949487686, -0.10433118045330048, 0.5934536457061768, -0.01793205365538597, -1.4482462406158447, 0.34008634090423584, -0.728383481502533, 0.20658794045448303], [2.3786964416503906, -1.1972578763961792, 1.1529417037963867, 1.064989686012268, -0.048671089112758636, 1.717551827430725, 0.43794864416122437, -1.4827772378921509, -0.5434406399726868, -0.6165170669555664, 0.028500398620963097, -0.8310363292694092, -0.5240365862846375, -0.022566765546798706, -0.19024814665317535, -0.8867903351783752, -2.2477810382843018], [0.31531473994255066, 1.758101463317871, 0.5013418793678284, -0.708259105682373, 0.11044298112392426, 0.426733136177063, 1.5454130172729492, 0.40946048498153687, -1.1270478963851929, 0.40875953435897827, -0.07855948060750961, 1.6593180894851685, -0.37378519773483276, 0.2662736773490906, 0.03542535752058029, 0.0399174764752388, -0.91751629114151], [-1.687503457069397, 2.613102674484253, -0.1863277703523636, -0.4170081317424774, -0.5487982034683228, 0.19211772084236145, 1.1780747175216675, 0.19635190069675446, 0.5242509245872498, 1.2632335424423218, 1.1755614280700684, -0.016408655792474747, 0.30751216411590576, 0.19709160923957825, -0.6212641000747681, -0.07706901431083679, 0.39851367473602295], [-0.2846904993057251, 2.2970449924468994, 1.2120611667633057, -1.0208473205566406, 2.2038540840148926, -1.102705955505371, 1.0377835035324097, -0.6945977807044983, -2.146059513092041, 0.2768089473247528, 1.155055046081543, -0.09060559421777725, -1.6108756065368652, -0.1012270376086235, -0.1949678659439087, 0.0053779808804392815, -0.23064197599887848], [1.3437808752059937, 0.4330407977104187, 1.8394842147827148, 0.6890377402305603, 1.0639755725860596, -1.6422380208969116, -0.7470014691352844, -1.2765495777130127, -1.5284305810928345, -0.46104854345321655, 1.3235352039337158, 0.23928944766521454, 0.24408724904060364, 0.05799121782183647, -1.15853750705719, -0.05135546997189522, 1.3009729385375977], [-0.2050228714942932, -0.06027243658900261, -1.633353590965271, 0.45667603611946106, -1.6990565061569214, 1.241058588027954, 1.5405910015106201, 0.1282820850610733, -0.32902804017066956, -0.24930498003959656, 0.213233083486557, 1.7921702861785889, -0.4504992961883545, 0.6793418526649475, -1.261631965637207, 0.7661405205726624, 0.6707945466041565], [1.2107548713684082, -0.4250437617301941, 1.3101245164871216, 0.1458914428949356, 1.6969423294067383, -0.8579797148704529, -0.19487859308719635, -1.2683011293411255, -0.7658345699310303, 0.938787579536438, -2.3236801624298096, -1.3372386693954468, 0.5843588709831238, 0.011323662474751472, 0.5641165971755981, 2.022388458251953, -1.3246349096298218], [-2.1307616233825684, 1.2181212902069092, -0.6232385039329529, -0.21592815220355988, -0.43364569544792175, 0.46295806765556335, -1.2190232276916504, 0.7889562249183655, 0.6296269297599792, 0.29008749127388, -1.655004858970642, -0.5302537679672241, -0.525484561920166, -0.7349266409873962, -0.0025130894500762224, -0.5914382338523865, -1.2837153673171997], [0.10243826359510422, -0.5946186780929565, 2.155684232711792, -0.10176736116409302, -1.2503243684768677, -1.111928939819336, 0.3924817144870758, 0.44831883907318115, -0.28608158230781555, 0.1353902667760849, -1.2440483570098877, 0.4869375228881836, -1.5709617137908936, -1.6085127592086792, 0.4332122802734375, -1.068017601966858, 1.0667946338653564], [-0.3970763385295868, -1.5396449565887451, 1.1946724653244019, -1.1358134746551514, 0.3637756109237671, -1.2754693031311035, -0.5201610922813416, -0.6628342270851135, 0.09544891119003296, -0.18033617734909058, 0.1606024205684662, 0.2840487062931061, -0.46674060821533203, 0.5003150105476379, 1.251311182975769, -0.06664037704467773, 1.4643850326538086], [-0.020457208156585693, -0.0613243505358696, 0.12812528014183044, 0.03584381937980652, -0.18042713403701782, 1.3613158464431763, -0.3728501498699188, 1.6830705404281616, 0.20632074773311615, 0.6365137100219727, 0.48926272988319397, 0.17669913172721863, -0.7057225704193115, -0.24634233117103577, -0.7014223337173462, -0.15578970313072205, 1.6026681661605835], [0.5453522801399231, 1.2852164506912231, -0.11975549161434174, 1.0654270648956299, 0.031606245785951614, -0.9816277623176575, 0.9798341989517212, 0.06887820363044739, -0.7442349791526794, 0.059219520539045334, -0.6481795310974121, 0.2400660663843155, -1.7132543325424194, -1.3913823366165161, -0.5995869636535645, 0.458993524312973, -1.2292237281799316], [1.0533424615859985, 0.5705344676971436, 1.0867582559585571, 0.04097215086221695, -2.7006163597106934, -0.5491150617599487, -1.657809853553772, -2.2320797443389893, -0.46209847927093506, -0.5620291829109192, 0.8384023308753967, 0.735530436038971, 0.1286085993051529, -0.5678592324256897, -1.4462262392044067, -0.4934144914150238, 0.5124354362487793], [-1.2354333400726318, -0.5998887419700623, -0.023968469351530075, -1.364341139793396, -1.696668267250061, -0.25475427508354187, 0.3855692744255066, 0.014323743060231209, 0.9857575297355652, 0.007334176450967789, 0.7086008787155151, -1.7841295003890991, 1.8468507528305054, -0.31554263830184937, -0.9204353094100952, -1.1052995920181274, -0.960828423500061], [1.5286132097244263, 0.12681451439857483, -0.9281335473060608, 0.09016984701156616, -0.1833452433347702, -1.3882558345794678, 0.9881941080093384, 0.1631752997636795, 1.0742779970169067, -0.24224653840065002, -1.7162742614746094, 0.7242648005485535, -0.8751969337463379, -1.2837409973144531, 0.6420751810073853, 0.9283344745635986, 1.193878412246704], [1.3010241985321045, 0.8197917342185974, -1.5885581970214844, -1.0883184671401978, 0.037897251546382904, -0.7433139681816101, -0.7672135829925537, -0.6649128198623657, -0.6440770030021667, -0.5977039337158203, 1.7905051708221436, -0.9407310485839844, 0.24629224836826324, -0.25345104932785034, 0.1549185961484909, 1.3132303953170776, -1.4043301343917847], [0.3854869306087494, -1.0527411699295044, -0.0947466641664505, 1.05022394657135, -0.523430585861206, 1.4106824398040771, -0.8915884494781494, 1.2941852807998657, -1.8140507936477661, 1.2209872007369995, 0.10166861116886139, 0.21519720554351807, 0.3604837954044342, 0.6744385361671448, 0.613481342792511, 0.053662337362766266, -0.8125741481781006], [0.5423743724822998, 0.565264105796814, 0.04730105400085449, -1.3105220794677734, -0.5338320136070251, -2.789202928543091, -0.18879354000091553, -1.2259324789047241, 1.8385653495788574, 1.1078053712844849, -0.8135021924972534, -1.1178863048553467, -0.11272002756595612, -0.9599761962890625, -0.7775033116340637, 1.1206316947937012, -2.6884350776672363], [0.10277339816093445, 1.8288999795913696, 2.068392753601074, -0.704857349395752, -0.3200325667858124, -1.2558964490890503, 0.29113373160362244, -0.01622192934155464, 1.3491040468215942, 0.6172799468040466, -0.23586627840995789, -0.5691556334495544, -0.27514609694480896, -0.12166484445333481, 1.5303972959518433, 0.4419249892234802, 0.48956719040870667], [-2.186096429824829, 1.0534968376159668, -0.25305280089378357, 0.8060175180435181, 1.2274093627929688, -0.0008510410552844405, 1.491543173789978, -0.8816750645637512, 1.696333408355713, 0.16078540682792664, 0.4339531362056732, 0.1467941403388977, -1.7603741884231567, 0.2779839336872101, 0.24647922813892365, 1.4422121047973633, 1.1657617092132568], [1.4808330535888672, -1.386287808418274, 1.0491585731506348, 0.18258802592754364, 0.3357796370983124, 0.18242233991622925, 0.2023291438817978, -1.1283349990844727, 0.32669275999069214, -0.35233408212661743, 2.6234586238861084, 1.5494552850723267, -0.7050926089286804, -1.6961427927017212, -1.0745925903320312, -0.22991327941417694, 0.5019926428794861], [-1.1000117063522339, 0.45351657271385193, -0.7509384751319885, 1.1366441249847412, 1.5993648767471313, -1.2034871578216553, 0.45696118474006653, -0.40474700927734375, -0.2004905790090561, -0.21367645263671875, -0.08831772208213806, -0.9582767486572266, 0.9827984571456909, 1.0072697401046753, 0.28671902418136597, -0.853717565536499, 0.18916691839694977], [-0.9313383102416992, -0.9532653093338013, -0.7379040718078613, 0.7008114457130432, 0.1156764104962349, -0.40657204389572144, 0.11587127298116684, -0.6282998323440552, 0.7133959531784058, -0.7343884706497192, 0.7419416904449463, -1.4725377559661865, -0.841047465801239, -1.096371054649353, -1.1558175086975098, -0.7514768242835999, 0.380275696516037], [-0.5149244070053101, 0.22082993388175964, 0.07988791912794113, -0.047460176050662994, -0.3366342782974243, 1.0821771621704102, 0.7403718829154968, -0.651527464389801, -0.36729007959365845, -0.2942964732646942, -0.04369570314884186, 0.7557254433631897, -1.589340090751648, 1.3998507261276245, -1.1621960401535034, 0.021636568009853363, 0.7659302353858948], [0.4639703929424286, 0.23129355907440186, 0.124876469373703, 0.17829103767871857, 2.6908276081085205, 0.43472346663475037, 0.4686521589756012, 0.347241073846817, 0.3758569359779358, -0.24051763117313385, 0.7319198250770569, -0.6144883632659912, -1.2457751035690308, -1.0172489881515503, -0.9650259017944336, -0.04878974333405495, -1.2425413131713867], [1.049284815788269, -1.1972720623016357, -0.8075218200683594, -0.6302669048309326, -1.5644772052764893, 0.45081546902656555, 1.6672589778900146, 0.005548779387027025, -0.1125231608748436, 0.8711233735084534, 0.4289815127849579, 0.36979031562805176, 0.24719087779521942, 0.3349284827709198, -0.694877564907074, 0.5356014370918274, 0.47052258253097534], [-0.12113496661186218, 1.5372332334518433, -0.7646893858909607, -2.9369008541107178, -0.1066652238368988, -0.10296343266963959, -1.8680928945541382, 1.6345432996749878, -0.44566401839256287, 1.4785370826721191, -0.1275445520877838, 1.1082329750061035, -2.871751070022583, 1.0952438116073608, -0.9598350524902344, 0.5001391172409058, -0.8169793486595154], [-1.4209673404693604, 0.18562443554401398, -0.693977952003479, -0.5094737410545349, -0.03408953174948692, 0.9550567269325256, -0.8601260185241699, 0.10845251381397247, 0.20930039882659912, 0.720599353313446, -0.6290327906608582, 0.2859703302383423, 0.0641264021396637, 0.24084582924842834, 0.4731031656265259, -0.53093022108078, -1.0725698471069336], [0.1994687020778656, -0.3364963233470917, 0.5145580172538757, -0.6708177924156189, 0.7322083115577698, -0.1274653673171997, -0.13629835844039917, 0.5115113854408264, -1.0298994779586792, -1.7694849967956543, 0.34375447034835815, 1.6708288192749023, -0.554891049861908, 0.044291846454143524, 0.7003592252731323, 0.5239881277084351, 0.19747143983840942], [0.9351748824119568, 0.229660764336586, 1.3848084211349487, 0.23783929646015167, 0.12659719586372375, -2.0772905349731445, 0.06937756389379501, -0.022714121267199516, 1.6407215595245361, -1.4807010889053345, -0.1732054352760315, 0.38134729862213135, -0.8147321343421936, -0.09452193230390549, 0.3456730246543884, -0.5831746459007263, 0.08406957983970642], [2.0283050537109375, 1.3184500932693481, 0.5347033739089966, -0.7495762705802917, 0.08782866597175598, 0.534633219242096, -0.6877064108848572, -0.7156851887702942, 0.05367428436875343, 0.8427005410194397, 0.4009776711463928, 1.1547352075576782, -1.1858443021774292, -1.381312608718872, -0.3539639711380005, -0.2762362062931061, 1.321637749671936], [1.6100138425827026, -0.16190437972545624, -0.7647536396980286, 1.3238452672958374, 2.8926024436950684, 0.715836226940155, 0.8039458394050598, -0.005698160268366337, -1.035732388496399, -0.3305980861186981, -0.7802473306655884, 0.7831429243087769, -1.5666154623031616, -0.40684986114501953, 1.6226117610931396, -0.2164621353149414, 0.9715405106544495], [-1.1160966157913208, -0.5825216174125671, -1.111467719078064, -0.11764385551214218, -1.7092241048812866, -0.5125839114189148, -0.9985748529434204, 0.4707852602005005, -0.7527015209197998, -0.12900851666927338, 0.1630316525697708, -1.155045509338379, -1.071523904800415, 1.2418667078018188, -2.1727194786071777, -1.8653216361999512, -0.15455292165279388], [2.7512869834899902, -1.9372590780258179, 0.5922675132751465, 1.2970976829528809, -0.8733184337615967, -0.5151703357696533, -0.8175092935562134, -1.121994972229004, 0.10182899236679077, 0.9522193670272827, 1.060258388519287, -2.5451595783233643, -1.1509106159210205, -0.46781596541404724, 1.3623020648956299, -0.6822049617767334, 0.44857001304626465], [-0.332083135843277, 0.7267149090766907, -0.126585453748703, -0.920997142791748, -0.7848505973815918, 0.8020000457763672, 0.05951419845223427, 1.4693020582199097, 0.21612460911273956, 0.6000993847846985, -0.8851540088653564, -0.9606977701187134, 1.014291524887085, 0.042875491082668304, -1.2377063035964966, -0.589137077331543, -0.08909022808074951], [-0.6364879608154297, -0.42229172587394714, -2.505033493041992, -1.1796222925186157, -1.9396209716796875, -0.8010203242301941, 0.335249125957489, 1.005152940750122, 2.567722797393799, -0.3540274500846863, -0.8111481666564941, 2.3767967224121094, -1.603798747062683, -0.09366074949502945, 1.0363696813583374, 0.5343856811523438, 0.6510676741600037], [-0.3152589201927185, 1.5483253002166748, 0.8128883242607117, 0.09678857028484344, -1.3897686004638672, 0.09785543382167816, 0.5756537914276123, -0.20253124833106995, 0.3357952833175659, 1.1116822957992554, -2.1382570266723633, -0.23759856820106506, -0.2300281822681427, 1.544724702835083, 0.2365710437297821, -1.9832600355148315, 1.099792242050171], [-0.42179155349731445, 1.460004448890686, 0.15826725959777832, 1.7041171789169312, 0.46330052614212036, 0.9296228885650635, -0.19734494388103485, -0.9747186899185181, 1.5474588871002197, 0.7642502188682556, -0.22773949801921844, 0.4458349943161011, 0.4177948534488678, -0.13026611506938934, -2.220210075378418, 0.13302700221538544, 0.7098245024681091], [-2.176701784133911, -0.7980340123176575, 0.058898426592350006, 1.2307049036026, 2.8884081840515137, 1.1260707378387451, 1.1944791078567505, -0.7673174142837524, -0.20325514674186707, -0.37020111083984375, 1.9123296737670898, -1.768685221672058, 1.2391331195831299, 0.9769315123558044, -1.1501553058624268, 0.43312010169029236, -0.6567241549491882], [-0.29151642322540283, -0.21420329809188843, -0.9390551447868347, 0.8025184869766235, 0.12512201070785522, -0.05988403782248497, 0.34715038537979126, -0.21706320345401764, -0.19938592612743378, 0.005454654339700937, -1.2548389434814453, -1.716313123703003, 0.8802617788314819, -0.5528793931007385, -0.6233996748924255, 0.025474606081843376, 1.6686609983444214], [-1.2797455787658691, -0.34266799688339233, -0.3763267397880554, -0.8848995566368103, -1.2811343669891357, 1.6450526714324951, -1.3959957361221313, -0.04754379391670227, -0.844287097454071, 0.6968810558319092, 0.22581970691680908, -0.936504602432251, 0.4063006341457367, -0.5796327590942383, 0.3641769587993622, 0.23521345853805542, 0.6176095604896545], [0.8739848732948303, -0.1873365342617035, -0.06454966217279434, 0.9336938261985779, -0.9134408831596375, -0.7407416701316833, 0.7584004998207092, -0.932811975479126, 0.15033818781375885, 0.442145973443985, -0.34672290086746216, -0.29366427659988403, 0.03244596719741821, -0.6141630411148071, -0.0424947552382946, 1.4342979192733765, -0.10992763936519623], [1.6616017818450928, -1.305641531944275, 0.5670313239097595, 0.24473907053470612, -0.4319538176059723, 1.798158049583435, -0.8920433521270752, -2.1574904918670654, 1.371697187423706, -0.12212874740362167, -1.3611284494400024, -0.2498781681060791, -0.20647874474525452, 0.5825944542884827, 0.15919412672519684, -0.3223913311958313, -0.9035754799842834], [-1.0489649772644043, -0.30882102251052856, 0.22438062727451324, 0.8232473134994507, -1.0586572885513306, -0.002366009633988142, -1.5730773210525513, 0.8573245406150818, 1.2609200477600098, 0.3290509283542633, -0.20043715834617615, -0.4078752100467682, 0.5576678514480591, -1.185144305229187, -0.5369616150856018, -0.5105100870132446, 0.9614812135696411], [0.57161945104599, -1.0343412160873413, 1.4022140502929688, -0.8343208432197571, 0.18318238854408264, 0.6379681825637817, -1.0097413063049316, 1.3170174360275269, 0.7730497121810913, 1.065929651260376, 0.37225091457366943, -0.12010708451271057, 0.7534464597702026, -0.46513837575912476, 0.28840121626853943, -0.2041114717721939, 0.006489220540970564], [0.375057578086853, 0.010520855896174908, -1.2084565162658691, -0.51075679063797, 0.24442534148693085, 0.45897117257118225, -0.6627897024154663, -0.8719987273216248, 1.216078758239746, -2.0092098712921143, 1.752699375152588, 0.3954777419567108, -0.5684427618980408, 2.4776127338409424, -0.10539454966783524, -1.3590199947357178, -0.19803304970264435], [-0.38560494780540466, -0.019121864810585976, 0.18495765328407288, 2.1249144077301025, 0.5006149411201477, 0.7094712257385254, 1.0118495225906372, -0.1529197096824646, -0.4322441518306732, -0.31761375069618225, 1.5601320266723633, 0.6901642680168152, 0.03820152208209038, -0.851690411567688, 1.2860617637634277, -0.276666522026062, -1.321205973625183], [-0.6404352784156799, -1.5360822677612305, -1.3963556289672852, -0.9779545664787292, -0.3575684130191803, -1.3922120332717896, -0.23424437642097473, 1.6683945655822754, 0.6559610366821289, -0.923723042011261, 1.528888463973999, 0.11121656745672226, -0.5406250357627869, -0.8593465089797974, -0.5048207640647888, -0.24306590855121613, -1.366616129875183], [-1.9554121494293213, -0.2219439148902893, -1.579771876335144, 0.5590677261352539, -0.4074021875858307, -0.13985678553581238, -0.4007887542247772, 0.10747414827346802, 0.48498424887657166, 0.43371134996414185, 0.25980231165885925, 1.2358465194702148, -0.5643186569213867, -0.15604184567928314, 0.5359367728233337, -0.3086407780647278, 1.8070915937423706], [0.5465666651725769, 0.8593081831932068, -0.7461158037185669, 0.3615111708641052, -0.5227137804031372, -0.9048061966896057, -1.4882194995880127, -0.27415257692337036, 0.07886361330747604, -0.9691709876060486, 0.47177204489707947, -0.052329905331134796, 1.2710134983062744, 0.6251799464225769, -1.2824398279190063, -0.053920406848192215, 1.7174190282821655], [-1.243515968322754, 0.8730889558792114, 0.1629469394683838, -0.4683345556259155, 0.47129085659980774, 0.041653476655483246, 1.0370521545410156, 0.2411205768585205, -0.8121656179428101, -0.26530295610427856, 0.6183973550796509, -0.6014243364334106, -0.7941669821739197, -1.4503817558288574, -0.8081186413764954, -0.8627476096153259, -0.5781338810920715], [1.526419758796692, 1.1606497764587402, -1.6826149225234985, -0.02390695922076702, -0.15424059331417084, -0.7610580325126648, 0.09465585649013519, 1.5810165405273438, -2.3935048580169678, -1.6766161918640137, 0.33891797065734863, -1.0408979654312134, -0.11015670746564865, 0.7866623401641846, 0.7540847063064575, -1.0948835611343384, 1.0130538940429688], [-0.1704408824443817, 0.8887320160865784, 1.583200454711914, -1.64287269115448, 1.3551169633865356, 2.2551066875457764, 0.41205278038978577, -0.28480055928230286, -1.3413456678390503, -1.2838677167892456, -0.8288999795913696, 0.33069169521331787, 0.4911624491214752, -0.6955381035804749, 0.8363363742828369, -0.2395564317703247, -1.2079124450683594], [-0.47727659344673157, -1.4447648525238037, 1.1933945417404175, 0.00018325119162909687, 1.054963231086731, 0.4820095896720886, -1.3157479763031006, -0.039142172783613205, -0.12424667924642563, -1.291704773902893, 0.3347800672054291, 0.7125011682510376, -0.6772617697715759, 0.4356847405433655, -0.0671665221452713, -1.2587496042251587, -0.859757661819458], [0.9619491696357727, 0.4448918402194977, 0.8281419277191162, 0.25319617986679077, -1.8018527030944824, -1.1278960704803467, 0.33367058634757996, -2.1396427154541016, -0.6350672245025635, 0.37605801224708557, -0.35143840312957764, 0.3181753158569336, 0.4457833468914032, 0.5886116623878479, 1.540696620941162, 1.1284518241882324, -0.032844312489032745], [-0.8439973592758179, -0.829857587814331, -0.883521318435669, -0.5280978679656982, 0.9115377068519592, 0.48245230317115784, 0.05158960074186325, -0.2326575368642807, -0.41907933354377747, -0.3358692228794098, 1.1477702856063843, 0.28074032068252563, 0.07062442600727081, -1.1866050958633423, -0.8409727811813354, -2.3687095642089844, 0.3375813961029053], [0.6580766439437866, -1.2307780981063843, -0.1737818866968155, -0.47633928060531616, -1.2512016296386719, -0.6695556044578552, 0.5117449760437012, -0.7952751517295837, -0.7969585061073303, -1.8173692226409912, -0.6847515106201172, 1.4425779581069946, -0.3947666585445404, 0.3378917872905731, 1.2850205898284912, -0.5294502973556519, 0.467352032661438], [2.62100887298584, 0.9470324516296387, -0.7136073112487793, 1.2700085639953613, 1.47739839553833, 0.6987438797950745, -0.434126079082489, -0.9185787439346313, 0.896036684513092, 0.4020402133464813, -1.2578637599945068, 0.5985245108604431, 1.3238024711608887, -0.40515977144241333, -0.33177322149276733, 0.9817531704902649, -1.546433925628662], [-0.4583596885204315, -1.7498717308044434, 0.09434805065393448, -0.07883916050195694, 0.08949203789234161, -0.7604253888130188, 0.5850613713264465, -0.06787668168544769, -0.25884702801704407, -1.1315083503723145, -0.49655431509017944, -0.07443510740995407, 1.2737425565719604, -0.8748380541801453, -0.30942824482917786, 0.05772138386964798, 0.007608778774738312], [-0.47638094425201416, 0.4270796477794647, -0.08820495009422302, -2.0952870845794678, -0.2225666642189026, -0.6544822454452515, -0.8052011728286743, -0.00948595255613327, 0.8808619976043701, 1.2902958393096924, 2.2590219974517822, -0.2853439450263977, 0.4397737979888916, -0.03406248614192009, -0.5034254789352417, -2.3152287006378174, 0.29067742824554443], [0.2595195174217224, -0.5566104054450989, 0.26415520906448364, 2.6197376251220703, -0.32981741428375244, -0.49049437046051025, 0.3067958354949951, -1.0692623853683472, 1.8145970106124878, 0.7573584318161011, -0.8448881506919861, -0.008036098442971706, 0.46580225229263306, -0.5965745449066162, 3.0027010440826416, -1.3534756898880005, -1.5221564769744873], [-0.9895408749580383, -0.6553353071212769, -1.3722295761108398, -1.1786952018737793, 1.0607424974441528, -1.2618671655654907, 0.4225277900695801, -0.022749783471226692, -0.2835633158683777, 0.6470729112625122, -0.9559081792831421, -1.9415289163589478, 1.9204329252243042, -0.1593845784664154, 0.2662905156612396, 0.08855404704809189, -0.9237750172615051], [0.0888998955488205, -0.7437640428543091, -2.215259075164795, 0.7161192297935486, 1.2915600538253784, 0.17340856790542603, -1.1161104440689087, 1.4523264169692993, -0.9348430633544922, -2.2154242992401123, 0.21381154656410217, -0.6509153842926025, 0.07329963147640228, -1.0033743381500244, 1.7990130186080933, -1.016350269317627, -1.4011948108673096], [0.6642905473709106, -0.7555513381958008, -0.894763171672821, 0.8855863809585571, 0.4407365024089813, 0.869116485118866, -0.28214380145072937, -0.24553337693214417, 1.5652766227722168, -1.9245030879974365, 0.18766522407531738, 0.9164974093437195, -1.4478175640106201, 0.9174590110778809, -1.4315979480743408, -1.3317915201187134, -0.250332772731781], [-0.03598461300134659, -1.1352921724319458, 1.3770455121994019, -0.0849405825138092, 1.0795140266418457, 0.6765693426132202, 1.5822153091430664, -0.3708590269088745, 0.5685697197914124, -0.5369656085968018, -0.4983765184879303, 0.8058590888977051, -0.1287812441587448, 0.34103864431381226, -0.8226513266563416, 0.91826993227005, 1.4777941703796387], [1.6267457008361816, -1.7405023574829102, 0.020512493327260017, -1.7028532028198242, -2.076326370239258, -0.4975728988647461, -1.5265969038009644, 0.19584153592586517, 0.8140736818313599, -0.43781203031539917, 0.2506435215473175, -0.03003513813018799, 0.5679177641868591, -0.09141513705253601, 0.7540203928947449, 1.6793924570083618, 0.719721794128418], [-0.7839013338088989, 0.05348014086484909, -0.4466582238674164, -1.1323164701461792, 1.3465182781219482, -0.6512587070465088, 0.2308545559644699, -1.3384023904800415, -0.8977944254875183, 1.3065234422683716, -2.0262644290924072, -0.32644888758659363, -0.7749391198158264, 1.2620705366134644, -0.9178681373596191, 0.9201161861419678, -0.9492034912109375], [0.07412981241941452, 0.26467636227607727, -2.7160656452178955, -1.0463659763336182, 0.11501159518957138, 0.25525903701782227, 0.04189523681998253, -0.26033464074134827, -0.013377044349908829, 0.6331267356872559, 1.0956321954727173, -0.8090346455574036, 0.2767163813114166, -0.1637106090784073, 1.6681369543075562, 1.6543922424316406, -0.20520859956741333], [-1.9028868675231934, -2.9747745990753174, -0.18331791460514069, -1.0970172882080078, 1.3570411205291748, 0.07079000025987625, -0.20762303471565247, 0.42152610421180725, 1.2167482376098633, -1.4559171199798584, 1.2340785264968872, -0.027846328914165497, 1.605155110359192, 0.7592152953147888, 0.08245394378900528, -0.009028321132063866, 1.5368132591247559], [-0.753395676612854, -0.15510304272174835, 0.947187602519989, 0.8380863070487976, -0.18190476298332214, -1.2024235725402832, 0.41617655754089355, -2.1502559185028076, 0.7429255843162537, 0.7742873430252075, -0.5166974663734436, 1.8173444271087646, -1.3017104864120483, 0.9634785652160645, 0.19015799462795258, -1.9441115856170654, 1.1098018884658813], [-0.7852895855903625, -0.4346652925014496, 0.2361489087343216, 0.8248984813690186, -0.9998565316200256, -2.373826503753662, 1.491034746170044, -0.5844477415084839, 0.41166990995407104, 0.6193751096725464, 1.0121777057647705, -0.022320488467812538, 0.6873466372489929, -0.45957762002944946, -0.5947875380516052, 1.9762471914291382, 0.006079334300011396], [-1.4418261051177979, -0.9551916718482971, -0.5435184240341187, -0.27763715386390686, 0.8864505290985107, -1.368277668952942, 0.26871541142463684, 1.1198476552963257, -0.9577955007553101, -0.4251922369003296, 1.7281055450439453, 0.1433304250240326, 0.10404956340789795, 0.14194715023040771, -1.5600001811981201, -0.3153892159461975, 0.8987579345703125], [-0.48835721611976624, 0.6311634182929993, 0.473855197429657, 0.9701211452484131, -1.5532256364822388, 0.6182636618614197, -1.495106816291809, -0.5711286067962646, 0.9149549007415771, 1.0950711965560913, -1.144110083580017, 1.122830867767334, -0.7226922512054443, -1.196712613105774, 0.16683146357536316, 0.8406156897544861, -0.02848154306411743], [-0.31152528524398804, 1.3473016023635864, -0.05970942974090576, -0.7543588876724243, 0.09293393045663834, 0.7233840823173523, 0.6209200024604797, -1.0612293481826782, -1.4160047769546509, -1.6324903964996338, 0.3546339273452759, -0.0005026544677093625, -0.5738065838813782, 0.9579771757125854, 1.857496738433838, -2.5672783851623535, 0.26478520035743713], [0.05259154736995697, -0.5309226512908936, 0.385387122631073, -0.6741525530815125, 0.7832760214805603, 0.35225775837898254, 0.18635177612304688, -0.13583536446094513, -0.5364830493927002, 0.09433404356241226, -1.4820985794067383, 1.8369663953781128, -2.1570746898651123, 0.7506394982337952, -0.7555105090141296, -0.8665457963943481, 0.5526643395423889], [0.1459471434354782, 0.09394263476133347, -0.0018398399697616696, 0.1913502961397171, 1.431054949760437, 0.6685458421707153, 0.2331056147813797, 1.2442196607589722, -1.3387155532836914, -1.4743289947509766, 1.137701392173767, 0.19689758121967316, 0.25089704990386963, 0.3131219446659088, -1.8500882387161255, 0.6350843906402588, -0.40445175766944885], [-0.9717583656311035, 0.8860097527503967, -0.32057562470436096, 1.6671360731124878, 0.605687141418457, -1.0181766748428345, -1.2843579053878784, 1.3195064067840576, -1.5514006614685059, 1.4039506912231445, -0.300491064786911, -0.8339579701423645, 1.2189853191375732, 0.6806063652038574, -0.3478953540325165, -0.911935567855835, -1.2908401489257812], [0.4328013062477112, 0.09911122918128967, -0.7677075266838074, -0.581227719783783, 0.26753199100494385, -1.150564193725586, -0.5426101088523865, -0.6499863266944885, 0.16321703791618347, -0.20171335339546204, -1.0866127014160156, 1.618523120880127, 0.6286095976829529, -0.6048390865325928, -1.6002973318099976, 0.2573834955692291, 0.3062826097011566], [0.5129079818725586, -0.5663003325462341, 0.3453654944896698, 0.39238637685775757, 0.9043570160865784, 1.89747953414917, 1.306336760520935, 0.12138289958238602, 0.17225737869739532, -0.6864575147628784, -0.18640591204166412, 0.4462131857872009, 1.110209345817566, -0.5977874398231506, 1.395591378211975, 0.14332999289035797, 0.17532186210155487], [0.9050928950309753, 0.4093606173992157, 2.0656397342681885, 0.5542577505111694, 0.05676289647817612, -0.4578821659088135, -0.8270651698112488, 1.3268333673477173, -0.16284120082855225, 0.31756392121315, -0.4835454821586609, -1.4433436393737793, -0.36144155263900757, -0.9053317308425903, 0.2599960267543793, 0.15968820452690125, 0.3190274238586426], [0.5495922565460205, 0.5568457245826721, -0.3124106824398041, 1.52369225025177, -0.32391077280044556, -0.07537370175123215, -0.7352025508880615, 0.027932051569223404, -1.537671446800232, -0.5675797462463379, -0.47644495964050293, -0.06677417457103729, -0.36275747418403625, 0.8419752717018127, 0.556473433971405, 0.014434270560741425, -0.37398678064346313], [-1.5476175546646118, 1.2962921857833862, 0.2446412742137909, -1.7027935981750488, 0.3062642514705658, 0.7451441884040833, -0.4032646417617798, 0.9133993983268738, -1.0678813457489014, 1.3416235446929932, 1.0872300863265991, 0.2253807783126831, -0.6309454441070557, 0.8831932544708252, 0.1400664746761322, -1.349767804145813, -0.0750696212053299], [1.3001604080200195, 0.13214637339115143, -0.44345107674598694, -0.5448046326637268, -0.6457799673080444, 2.0314087867736816, 0.33901458978652954, 1.7089734077453613, -1.1147769689559937, -0.5026800632476807, -2.023425340652466, -0.3452145755290985, 0.08106706291437149, -0.5701788663864136, 0.7069382071495056, -0.49545690417289734, -0.11204107850790024], [0.017931848764419556, -0.7055455446243286, -1.4733926057815552, -0.3467899262905121, 0.02703351527452469, -1.4556697607040405, -0.47587043046951294, -0.21149633824825287, 1.8434628248214722, 0.14964190125465393, 0.9679144620895386, -0.12520839273929596, -1.1774818897247314, -0.15503954887390137, -1.3818665742874146, 1.3618278503417969, -0.19089068472385406], [0.0680253654718399, -0.29928964376449585, 0.6299460530281067, 1.3911449909210205, -0.10225251317024231, 0.30475708842277527, -0.5989887118339539, 0.4745063781738281, 1.2292819023132324, 0.35804983973503113, -0.215725377202034, -0.02289341762661934, 0.3045900762081146, 1.3507829904556274, 0.44668710231781006, -0.006037974264472723, -0.5810024738311768], [-0.9815399050712585, 1.7837470769882202, -0.3062712550163269, 0.6737626791000366, 1.3323626518249512, 0.00015099585289135575, -1.026576042175293, -0.001635671709664166, 1.323931336402893, -0.13650821149349213, -0.7482852935791016, 0.20315703749656677, 1.8609365224838257, -0.39247432351112366, -0.23903857171535492, 0.1331491321325302, 0.6356651186943054], [-1.1203633546829224, 1.1715073585510254, -0.874213695526123, 1.3238123655319214, -0.762671947479248, 1.2108107805252075, 1.198861002922058, -0.7472381591796875, -2.482541799545288, -1.6291484832763672, -0.055469512939453125, -1.067742943763733, -0.5307245850563049, -0.006261156871914864, -0.14596880972385406, -0.7994806170463562, -0.8481099009513855], [0.1527709811925888, 0.2640058398246765, 0.7832005620002747, -0.10599906742572784, 0.40360361337661743, 1.8186089992523193, -0.3294444680213928, -0.0630635991692543, -1.0335273742675781, 0.03189663961529732, -0.4729337990283966, -0.9072784781455994, -0.2957468032836914, 0.08625340461730957, 0.48916077613830566, 1.1756062507629395, -0.4320650100708008], [0.5401024222373962, 1.0524861812591553, 0.5576277375221252, 2.2810707092285156, 1.0163311958312988, 0.30268406867980957, -0.21813620626926422, 0.2607954144477844, 1.2818903923034668, -0.4192673861980438, 0.7387171983718872, -0.6186016798019409, -2.228245973587036, 0.5231527090072632, -0.14989325404167175, 0.6267683506011963, 2.047985315322876], [1.0202125310897827, -0.35538119077682495, -0.22695139050483704, -0.9677394032478333, 0.7962062954902649, 0.16556653380393982, -0.833712100982666, 1.077001690864563, 0.2839192748069763, 0.9370982646942139, 1.2397053241729736, -0.2262781709432602, -0.42811688780784607, 0.6866247653961182, 0.4696658253669739, -0.7813767790794373, 1.0207924842834473], [0.14959633350372314, 0.5136746168136597, -0.5591691732406616, 0.3719981908798218, 0.729476809501648, 1.185133457183838, -0.5941280722618103, -0.5190352201461792, 0.945231020450592, 0.775016725063324, 0.6053099036216736, -2.5908055305480957, -0.006751121953129768, 0.8234629034996033, 0.15328973531723022, -0.6237964034080505, -1.381385087966919], [0.8173694014549255, -1.0178699493408203, -0.9318624138832092, 0.5517750382423401, 0.3532852530479431, 0.7295223474502563, 2.644549608230591, 0.11409861594438553, -0.9890768527984619, -1.3641562461853027, -0.2838568389415741, 0.8854206204414368, -0.9991092681884766, 2.1171302795410156, -0.08372587710618973, 0.08714546263217926, 0.05808419734239578], [-0.29696333408355713, 0.4648212790489197, 0.1582942008972168, -0.14487971365451813, -2.31740403175354, 0.27921801805496216, -0.6084298491477966, -0.49137797951698303, 1.2524127960205078, 0.19567549228668213, 0.3725140392780304, -0.09952397644519806, -0.12206414341926575, -1.1623992919921875, 0.06044955551624298, -0.5929282903671265, 0.07617652416229248], [0.33051490783691406, 0.37686848640441895, -0.26307469606399536, 0.27965250611305237, -0.35253626108169556, -0.31825846433639526, 2.075650215148926, -1.5642316341400146, -0.48237255215644836, -0.3766418695449829, -1.3708858489990234, -0.662714421749115, -1.7120920419692993, 0.36049139499664307, 0.25382742285728455, -0.11789947748184204, 0.8914860486984253], [1.112345576286316, -0.2235308289527893, 1.184440016746521, 1.3576154708862305, 0.8252150416374207, -0.0018191636772826314, -0.9076829552650452, 1.0963877439498901, 2.1202099323272705, 0.29244571924209595, 1.476456880569458, 0.993911862373352, 0.7701097726821899, -2.079120635986328, 0.6823769807815552, -0.2806909680366516, 0.6521465182304382], [-0.11519046872854233, 1.531317949295044, 1.1525417566299438, -1.2353910207748413, 1.0650547742843628, -0.4266432225704193, 0.5166399478912354, -1.8128347396850586, 0.3701249361038208, 0.8171122074127197, 0.7241335511207581, 0.8630966544151306, 1.384139895439148, 0.09650116413831711, 0.21553343534469604, 0.24233180284500122, 0.4065231382846832], [0.20241962373256683, -0.3604657053947449, -0.302203893661499, 0.43739986419677734, 0.18232841789722443, 1.7617194652557373, 1.0214906930923462, 0.843660295009613, 0.24188145995140076, 0.9788596034049988, -0.9957647323608398, 0.09700927138328552, 0.08957431465387344, 1.4981063604354858, 0.05490446835756302, -0.9286118745803833, 1.0559723377227783], [2.3335323333740234, 0.6462065577507019, 0.054317161440849304, -0.8240169286727905, 0.12896470725536346, -2.951551914215088, -0.2890588343143463, 0.5200702548027039, 1.0563538074493408, 0.16942065954208374, -0.12801897525787354, 1.031084656715393, 0.043855633586645126, -0.5905596613883972, 0.3896791338920593, -0.33493131399154663, -0.7532573342323303], [-1.3129403591156006, -0.9797459244728088, 0.09920787811279297, -0.9884413480758667, -1.6614928245544434, 0.4403087794780731, 1.0791912078857422, 0.7751121520996094, 0.40908199548721313, 0.4218721091747284, -1.196165919303894, -0.29325512051582336, 0.5497487783432007, -1.072201132774353, -1.4440361261367798, 0.741077721118927, -0.31198152899742126], [-0.19808229804039001, -1.2107651233673096, 0.8119757771492004, 0.9393752813339233, -0.9275884032249451, -0.18818619847297668, -0.20597828924655914, -0.9586290121078491, -0.9229995608329773, 0.2225106954574585, -0.14182333648204803, -0.5555569529533386, 0.9865380525588989, -0.32094115018844604, 0.805984377861023, -0.2693381905555725, -0.7369760870933533], [-0.8921761512756348, -0.385643869638443, 0.08912089467048645, -1.1839444637298584, -0.013506574556231499, -1.314174771308899, -0.4454137682914734, -1.3780757188796997, -0.2606083154678345, -1.8364145755767822, 1.6181305646896362, 1.082929015159607, -0.2566767632961273, 0.12466072291135788, 0.9259222149848938, 0.24396546185016632, 1.8625469207763672], [0.12510675191879272, 1.1712586879730225, -0.4948408305644989, 1.8827664852142334, -0.2886884808540344, 0.1289401799440384, 1.752243161201477, 0.6180413961410522, 1.244743824005127, 0.517701268196106, 0.14424575865268707, 0.9809833765029907, -0.3344461917877197, 0.8448318243026733, -1.7938354015350342, 0.1669996976852417, 0.30564191937446594], [-1.5658881664276123, -1.1888197660446167, -0.8697506785392761, -0.8835598230361938, -0.28704899549484253, -1.2549865245819092, 0.06118445098400116, 1.448829174041748, -0.9627878069877625, 0.6305804252624512, -0.5235100984573364, -2.1943418979644775, 0.7782403230667114, 0.659496009349823, -1.4702260494232178, 0.459524929523468, 0.10019715875387192], [-2.3691580295562744, 0.1806311309337616, 0.7540993094444275, 2.134094715118408, -0.28548580408096313, -0.09015798568725586, 0.19521845877170563, -0.2707747220993042, 2.1011650562286377, -1.328391432762146, -0.3675559163093567, -1.1949936151504517, 0.38062578439712524, 0.7239296436309814, -1.1987149715423584, -0.877601146697998, -1.2659224271774292], [-0.44530990719795227, -1.098749041557312, -0.712889552116394, -0.38823944330215454, -1.2535868883132935, -0.3581617474555969, 1.4016153812408447, -0.01261279359459877, -0.7337482571601868, -0.3231714367866516, -0.25423431396484375, 1.057531714439392, 1.0371886491775513, 1.837422490119934, -0.9117918610572815, -0.7567219138145447, -0.34278568625450134], [0.3660138249397278, 0.7781224250793457, -1.8087074756622314, -1.1401149034500122, 3.482304811477661, 0.16431377828121185, -0.1405155211687088, 0.1712617427110672, 1.3734647035598755, -0.8002133369445801, -0.827284574508667, 1.5166717767715454, -1.894710898399353, 0.057695694267749786, -0.6318570375442505, 0.43596550822257996, -0.4638407230377197], [0.3414497673511505, -1.0407596826553345, 2.387488603591919, 0.2588121294975281, -1.191260576248169, -0.6448892951011658, 1.0684889554977417, 0.9875487089157104, 0.06666664779186249, 1.1900352239608765, -0.13677063584327698, -0.5972766280174255, -0.9240356683731079, 0.7653778195381165, -0.6149041056632996, -0.7210164070129395, -0.32540321350097656], [1.4297178983688354, 0.9904350638389587, -0.12033510208129883, 0.11779777705669403, 1.3138933181762695, -0.5668251514434814, -0.08067082613706589, -0.5174028873443604, -2.6931495666503906, -1.020813226699829, -0.042171794921159744, 0.2499920129776001, 0.01441131066530943, -1.0325653553009033, 0.05894340202212334, -0.1324397176504135, -0.6134627461433411], [0.20591242611408234, 0.25907212495803833, 0.28844985365867615, 0.2594604194164276, 0.7256941199302673, 1.3245831727981567, 0.5926752686500549, -0.10708452761173248, -0.7156504392623901, 0.12181168794631958, -0.12660786509513855, 2.8292441368103027, 0.08449050039052963, -0.31733712553977966, -0.6203901767730713, 1.0532718896865845, 1.4117891788482666], [0.06205437332391739, 0.007799685467034578, -0.17142091691493988, -0.5853021144866943, -1.210501790046692, 0.6969176530838013, 1.0108764171600342, 0.767693042755127, 1.1589372158050537, 0.07773162424564362, 0.05477394163608551, 1.1172819137573242, 1.4186692237854004, 0.49204111099243164, 0.6784014105796814, 1.4093105792999268, 0.022908339276909828], [-0.07740959525108337, 1.1291418075561523, 1.162459373474121, 1.2449284791946411, 1.184215784072876, -0.8935303688049316, -0.028742443770170212, 1.0219271183013916, -0.7288705110549927, -0.353362113237381, 1.0278271436691284, -0.8927172422409058, 0.6729040741920471, 0.564608097076416, 0.4129457473754883, 0.009649128653109074, -0.5635198354721069], [0.44022202491760254, 1.3715020418167114, -1.3008543252944946, 0.13902726769447327, 0.35722729563713074, 1.0103821754455566, 1.4393327236175537, 0.21280013024806976, 1.209420084953308, 1.5146257877349854, -0.44168317317962646, 0.6808466911315918, -1.1802150011062622, 1.5879573822021484, 0.2945910394191742, -0.028504516929388046, 0.24554458260536194], [2.0695955753326416, 1.413669228553772, -0.4949814975261688, -1.3984578847885132, 0.26807695627212524, -1.2030913829803467, -1.927169680595398, 2.5412044525146484, -0.6687489151954651, 0.6565732359886169, -0.6976838707923889, 2.5215225219726562, -1.7119098901748657, 1.3919286727905273, 0.6636725664138794, -0.9588897824287415, 0.4201115369796753], [0.33879151940345764, 0.4184721112251282, -0.6678667664527893, 0.348003089427948, -0.1294093281030655, 0.3728693425655365, 0.7315596342086792, -0.43639716506004333, -0.4939199984073639, -0.35416385531425476, -0.4739479720592499, -0.38663408160209656, 0.49216458201408386, 0.5762407183647156, -1.9316236972808838, 0.27510690689086914, -1.016148567199707], [0.8049278259277344, 0.36400312185287476, 1.1605943441390991, -0.8491435050964355, -0.49862802028656006, 0.5682893991470337, 0.7725323438644409, 0.7865126729011536, 0.9902597665786743, -0.3828524947166443, -1.6703596115112305, -0.28690892457962036, -1.251476526260376, 0.013012985698878765, -0.4088374674320221, -1.541813850402832, -1.3205442428588867], [0.7335922122001648, 1.2560408115386963, -1.908267617225647, -0.437212198972702, 0.31799814105033875, -0.3288094401359558, -0.0904579609632492, 1.03819739818573, -0.5711938738822937, 0.270168662071228, -0.21091164648532867, 0.665661096572876, -1.0425044298171997, 0.47065049409866333, 0.08222340047359467, 0.09378714859485626, 1.7739089727401733], [0.19474495947360992, -1.0291328430175781, -0.03556758165359497, -0.2512809634208679, -0.698691189289093, 1.075677514076233, -0.7931424975395203, -1.1950275897979736, -0.7941287159919739, -0.16172008216381073, -0.41913917660713196, -0.6808218955993652, -0.0009897856507450342, -2.1942639350891113, 0.25064617395401, -0.4132230579853058, 2.8291919231414795], [0.23347117006778717, 0.6743101477622986, -0.170029416680336, 0.292912095785141, 0.8942471146583557, 0.8571931719779968, -0.31222954392433167, -1.4934653043746948, -1.021460771560669, 0.5833951234817505, 0.43054425716400146, 0.26316267251968384, -0.6273847222328186, -0.29857850074768066, 0.20314215123653412, 0.8966553807258606, -0.038402821868658066], [0.019473833963274956, 0.17168472707271576, -0.028798138722777367, 1.24399995803833, 1.0121575593948364, -1.0182803869247437, -0.03128053992986679, -0.9110355377197266, 0.9246284365653992, 0.5178066492080688, -2.491420030593872, -0.7064403891563416, -1.4457396268844604, -0.07329966127872467, -0.5272546410560608, 0.4718005657196045, -1.0283960103988647], [-0.7517473101615906, -0.4980013370513916, 0.9157165884971619, 0.8914942741394043, -0.33982548117637634, 0.8118025660514832, -1.0746502876281738, -0.50785231590271, 1.4396029710769653, 0.8215967416763306, -1.5545861721038818, 0.33997780084609985, 0.5936313271522522, -0.6050024628639221, 1.4878584146499634, -0.5656996965408325, -0.42116034030914307], [0.019997835159301758, 0.7731886506080627, -0.2636438310146332, 0.4839802086353302, -0.4172777235507965, 0.4096950590610504, 0.454417884349823, -0.5756796598434448, 0.7504216432571411, -0.7330932021141052, -1.2561105489730835, 0.49358776211738586, 0.9193875789642334, -0.4448963701725006, 0.06253966689109802, -0.269260436296463, -0.45491164922714233], [0.6889126300811768, -1.4684818983078003, -2.1913087368011475, -0.769673228263855, -0.10914738476276398, 0.4511968493461609, -0.06732964515686035, 0.41727373003959656, -1.2817902565002441, -0.04893581569194794, 0.8144015073776245, -0.31985199451446533, 0.6457917094230652, -0.4355326294898987, 0.2901255786418915, -1.7199229001998901, 0.3500431180000305], [1.4551801681518555, -1.868514060974121, 1.6432914733886719, -0.9019328951835632, -0.6982235312461853, 0.031621478497982025, -0.6369116902351379, -0.5816749334335327, 1.2361721992492676, 0.2054130584001541, -0.24071139097213745, -0.35821157693862915, -0.6972757577896118, -0.9852694869041443, -0.9870122671127319, 0.08653667569160461, 1.7693595886230469], [-1.689935326576233, -1.122697114944458, 0.23588621616363525, 0.21883942186832428, -0.7913089394569397, 0.281389445066452, -1.4064640998840332, -0.03312317281961441, 0.07873004674911499, -2.2386984825134277, -2.3279435634613037, 1.0015134811401367, 1.8793070316314697, -0.1334940791130066, -0.43807950615882874, -0.205919548869133, 0.6193361878395081], [0.77585768699646, 0.42356735467910767, -0.22733230888843536, 0.2089150995016098, -1.0660457611083984, 0.2736922800540924, -0.7147671580314636, 0.39221659302711487, 2.6333770751953125, -1.180411458015442, 1.0201457738876343, -1.5953831672668457, 1.1420879364013672, 1.573103904724121, 0.7929461598396301, -0.3911204934120178, 0.2507123649120331], [-0.25872570276260376, -0.1249377503991127, 1.2566739320755005, 1.3051320314407349, 0.90189129114151, -1.038620114326477, -1.3436576128005981, 0.8717342615127563, -0.27619245648384094, 0.0120105454698205, 1.207706093788147, -0.38278552889823914, 0.3899048864841461, 0.3070487678050995, 1.7887904644012451, -0.7529460787773132, 2.027350902557373], [-1.3031253814697266, -0.15892711281776428, -1.0452229976654053, -1.0080533027648926, -0.6101567149162292, -0.15460282564163208, 0.8159792423248291, -0.39276251196861267, -0.5377118587493896, -0.5452995300292969, 1.5051894187927246, 0.2551427483558655, 2.3689000606536865, 0.8602308034896851, 1.4444180727005005, 0.0738363042473793, 0.1330767124891281], [-1.379656434059143, 0.2875928580760956, -0.22585582733154297, 0.08720675855875015, 0.8178972005844116, -0.9624971747398376, 0.828391432762146, -1.177839994430542, 1.3929977416992188, 0.35061225295066833, 0.7662015557289124, -0.07078094780445099, -0.2597183585166931, 1.1674354076385498, 1.25298273563385, -0.18421754240989685, 0.5736900568008423], [-0.05497715249657631, 2.38527512550354, -0.9815106987953186, 1.5788484811782837, 1.583850383758545, -0.011422157287597656, -1.254335880279541, 1.564669132232666, -0.38583284616470337, 0.8273440003395081, 0.718425989151001, -0.3473811149597168, 0.6168373227119446, 0.4328606128692627, 0.10946843028068542, 0.11201834678649902, -0.4384571611881256], [-0.10232659429311752, 1.5622787475585938, 0.9349765777587891, -0.7189491987228394, 0.5415232181549072, 0.2678658068180084, -0.617356538772583, -0.20760148763656616, -0.4727194309234619, -0.42079630494117737, 0.8102667927742004, 0.20082814991474152, 0.18853682279586792, -1.4322532415390015, 1.0967460870742798, -1.6710734367370605, 1.8958730697631836], [-0.009712434373795986, 0.9309072494506836, 1.175506830215454, -0.783585250377655, -0.09672864526510239, -0.5585167407989502, 1.3679239749908447, -0.9342091083526611, 0.7783480286598206, -0.6693536043167114, -0.403739333152771, -2.7173807621002197, -0.8158047199249268, -0.5799582600593567, 1.3964824676513672, -0.1762717217206955, -0.02275286614894867], [-0.8757562041282654, -0.3498021066188812, -2.537018299102783, -1.088770866394043, -0.8710977435112, -0.8526674509048462, -1.5222135782241821, -0.6912675499916077, -0.20222865045070648, 0.01871504820883274, 0.38864192366600037, 0.18002624809741974, -0.2507777810096741, 0.43808847665786743, -2.5517101287841797, 0.7351589202880859, 0.36184272170066833], [-1.450316309928894, -0.05797227472066879, -0.8718061447143555, -2.319871187210083, -1.8380587100982666, 0.9417065978050232, 0.38976579904556274, 0.8030043244361877, -1.1996204853057861, 0.5188664793968201, -0.7957068085670471, 1.2862176895141602, 1.1613988876342773, -0.9207162261009216, -1.1201558113098145, 0.32365357875823975, -1.0950016975402832], [-1.9626959562301636, -1.085884690284729, -1.4207419157028198, 0.7095892429351807, 1.1613637208938599, 0.9951112866401672, -1.375205159187317, 0.6385769248008728, -0.02689664624631405, -0.48819348216056824, -0.26402243971824646, 1.478630781173706, -0.28030160069465637, 1.4551712274551392, 1.9414856433868408, -0.8738329410552979, 0.06331151723861694], [0.016337336972355843, 0.22110886871814728, -0.9946900606155396, -0.7033848166465759, 0.6230767369270325, 0.2923847436904907, 1.1687655448913574, 0.05712734907865524, 0.33064404129981995, -2.039641857147217, 0.17810697853565216, 0.7022366523742676, 1.1618406772613525, -0.23789046704769135, 1.3586485385894775, 0.44575783610343933, 0.6001445055007935], [-0.3764152228832245, 0.8845999836921692, 0.34458351135253906, -0.6194460391998291, -0.31048068404197693, -1.0816162824630737, 0.45206886529922485, -2.0951364040374756, 1.3934823274612427, -1.202276587486267, 0.3422401249408722, 0.7116555571556091, -1.0653841495513916, 0.002828376367688179, -0.5256937146186829, -1.1619609594345093, -0.5169933438301086], [0.1594994217157364, -1.3158034086227417, -0.9630917906761169, -0.6471158862113953, -0.3358953297138214, 0.511600136756897, -0.5144429206848145, 0.11925698816776276, -0.08251836150884628, -1.594070553779602, 0.6744431257247925, -0.16016657650470734, -1.081708550453186, -0.2503719925880432, 0.8505403995513916, 0.6921833753585815, -0.5827395915985107], [-1.2643804550170898, 0.1862393021583557, 0.7795780301094055, -0.3543286621570587, 0.5492225885391235, 0.3196687400341034, -0.3418400287628174, -0.35418567061424255, -0.8350585699081421, 1.9038164615631104, 0.4664941132068634, -0.9842432141304016, -0.22850428521633148, -0.17954778671264648, 0.28491562604904175, 1.6011728048324585, 1.2733690738677979], [-0.42138785123825073, -1.6823501586914062, 0.6808542609214783, -0.932811975479126, -0.116672083735466, 0.3303726315498352, 0.6176967620849609, 0.2596573233604431, 0.6095067262649536, -1.0599961280822754, 1.80153226852417, 0.6451510190963745, -1.5590139627456665, -1.3816332817077637, 1.1865355968475342, 0.49849069118499756, -0.8992068767547607], [-0.5050157308578491, 0.6398399472236633, 1.3919780254364014, -0.6549199819564819, -1.4580492973327637, 2.742159366607666, 1.5198320150375366, -1.2796070575714111, -2.2202813625335693, 1.3934290409088135, 1.3737308979034424, -0.44227835536003113, 1.4259129762649536, -0.9850448369979858, 0.5331226587295532, -0.9624871611595154, 1.9965378046035767], [0.23228618502616882, 1.7887210845947266, 0.18774442374706268, 0.6075480580329895, 0.3163721263408661, -0.17718838155269623, -0.9175352454185486, 0.2860185205936432, 2.096200704574585, 0.39451926946640015, -0.6696661114692688, -1.2796376943588257, 0.34544721245765686, -0.17107123136520386, 0.2711299955844879, -2.8426175117492676, 1.1595078706741333], [0.9601916670799255, -0.06388339400291443, 2.2723946571350098, -0.39436936378479004, -0.7803788185119629, -0.4546980559825897, -1.0504485368728638, -0.20064157247543335, -0.011933212168514729, -0.9554029703140259, 0.507764995098114, -0.9896157383918762, -0.2873390018939972, -1.016633152961731, 0.9607394933700562, -0.06302663683891296, 0.5152885317802429], [1.2111144065856934, -0.2113901525735855, 0.7251843214035034, -0.6992477774620056, -0.9549461603164673, 0.3188452124595642, 0.2125106304883957, -1.8109976053237915, -2.3752567768096924, 0.6181325912475586, 0.7668296694755554, 1.5083335638046265, -1.675351619720459, -0.048167694360017776, 1.2436577081680298, 1.9475842714309692, 1.315116286277771], [1.6136817932128906, -0.02480107732117176, 0.937904417514801, 1.0072895288467407, -0.17265956103801727, 0.6840788722038269, 1.2681995630264282, 2.7567927837371826, -0.21899034082889557, -1.3234509229660034, -0.08853255212306976, -0.06414967030286789, 0.8713318109512329, -0.11752424389123917, -0.8721741437911987, 0.1313367635011673, -0.07977721840143204], [-0.12573370337486267, 0.6263516545295715, 1.2101258039474487, -0.8482753038406372, 0.5709599852561951, 1.464169979095459, -0.818011999130249, 1.1941194534301758, -0.11214689165353775, -0.024982469156384468, 0.16185155510902405, -1.3801960945129395, -0.9870873689651489, 1.043010950088501, -1.2305572032928467, -1.021562099456787, 0.4314923584461212], [-1.7499854564666748, 1.6973296403884888, 0.5625782012939453, 2.053525924682617, 0.40153151750564575, -0.9417214393615723, -0.5717612504959106, 1.1542147397994995, 0.7612211108207703, -0.9069209694862366, 0.6326571702957153, -0.7800770998001099, -2.6607542037963867, -0.22997666895389557, 0.8955442309379578, 1.5050796270370483, -0.0888211652636528], [-0.12621484696865082, 0.4940480887889862, 0.6422438025474548, 0.23409517109394073, 1.0479519367218018, 0.5971678495407104, 1.4421077966690063, 1.3965153694152832, -1.8021962642669678, 0.9263375401496887, -0.14338333904743195, -0.055594801902770996, 0.9122915863990784, -0.35301896929740906, 1.6112595796585083, -0.4917227029800415, -1.5008466243743896], [-1.160874843597412, 0.02483951672911644, -0.06388352066278458, -0.4007837474346161, -1.4801225662231445, -0.931038498878479, -0.21585339307785034, -0.9920414090156555, -1.0311063528060913, 0.25381359457969666, 0.5981982946395874, -0.730246901512146, -1.838846206665039, -0.48367637395858765, -0.751497209072113, -1.0924663543701172, 0.13591915369033813], [1.087942123413086, 2.5206472873687744, 0.20552703738212585, -0.25470083951950073, -1.7146527767181396, 0.5725414752960205, 0.73295658826828, -1.1204297542572021, 0.8119109272956848, 0.05411732196807861, 0.8233519792556763, 0.9398569464683533, 1.8991566896438599, -0.41861405968666077, 0.46636316180229187, -0.6575300097465515, -0.049888089299201965], [1.060752511024475, 0.984047532081604, 0.2118293046951294, -0.35629531741142273, 0.6909484267234802, -1.4305546283721924, 1.4538474082946777, 0.6440458297729492, -0.5684730410575867, -0.11185972392559052, -1.377247929573059, 0.028012976050376892, 0.050191376358270645, -0.23585057258605957, 0.825394332408905, 0.46314194798469543, -1.3455711603164673], [-0.17263446748256683, 0.4092289209365845, 0.35119742155075073, -0.008657381869852543, -0.6178846955299377, 0.8522946238517761, 0.2320321649312973, -0.23211762309074402, 0.04220087453722954, 0.1680009663105011, -0.2344730645418167, 1.1037184000015259, -1.5392063856124878, -2.1044671535491943, 0.8724395632743835, 0.8737990260124207, 1.8231502771377563], [-1.3507940769195557, 2.153439998626709, 0.36998748779296875, 0.17982186377048492, 0.81960529088974, 0.27390503883361816, -1.2418280839920044, 0.143886998295784, -0.7211558818817139, -1.177895188331604, -0.08912146091461182, 1.000616431236267, 0.23184694349765778, -0.5579850673675537, 1.203493356704712, 0.4466369152069092, -0.40708526968955994], [1.493064045906067, 0.26944684982299805, 0.7120261788368225, -1.4209916591644287, 0.679242730140686, 0.9083120822906494, 0.17859508097171783, 0.6112545132637024, 0.028278399258852005, -0.017434701323509216, 0.19754017889499664, -0.8472517132759094, 0.18253786861896515, 0.026516638696193695, 0.918006420135498, -0.7115055918693542, -1.0333648920059204], [-0.040168847888708115, -0.5297688245773315, -1.3378196954727173, 0.4061647355556488, -1.640246868133545, 0.5447350740432739, -0.0022088466212153435, 0.6818231344223022, -0.37855854630470276, -0.5928083658218384, 1.0314303636550903, -0.31714433431625366, 0.8526479601860046, -2.2589385509490967, 0.9210694432258606, -0.44990259408950806, 0.8961398005485535], [-0.7263805270195007, -0.4243614971637726, -0.21019311249256134, -0.04588976129889488, -1.4761239290237427, -0.07592616230249405, -0.7334055304527283, -0.18795084953308105, -0.04837033525109291, -1.514032244682312, 0.8219367861747742, -0.5991750359535217, -0.4994153082370758, 1.484364628791809, 0.7687126994132996, -0.4307318329811096, 0.7036885619163513], [0.6175965666770935, 1.0126100778579712, 0.4887089133262634, 1.4277524948120117, -1.793554425239563, -0.35245054960250854, -1.1049063205718994, -3.356125593185425, 1.0087963342666626, 1.3361179828643799, -0.8527301549911499, -0.26125580072402954, 1.0006309747695923, -0.22240017354488373, -0.4327535033226013, -0.45426270365715027, 0.09886560589075089], [-0.8720535635948181, -0.39104020595550537, -0.24995122849941254, -0.7176164984703064, -1.4604086875915527, 0.8596462607383728, -0.23328013718128204, -0.7684438824653625, 1.097173810005188, -0.7685289978981018, -0.17343275249004364, 1.351373553276062, -0.04766459763050079, -0.059744954109191895, 0.5148481130599976, -2.048901319503784, -1.4888722896575928], [0.6319422125816345, -1.2327466011047363, 1.1020227670669556, -0.2267727255821228, 0.6731635928153992, -0.9040553569793701, -0.39904820919036865, 0.9518329501152039, -0.3833673298358917, 0.40270674228668213, -0.6336567401885986, 0.06473943591117859, -2.1119487285614014, -0.3321591019630432, -0.6745533347129822, 1.4157241582870483, -0.26313328742980957], [1.375942587852478, -0.27286672592163086, 0.5861599445343018, -0.07377228885889053, -0.7245743274688721, 0.07549310475587845, 0.4120001792907715, -0.5042098164558411, 0.2348961979150772, 0.7174493074417114, -0.7874688506126404, 0.19256992638111115, 0.4215419292449951, 0.48150455951690674, -0.4054650366306305, -1.0881891250610352, 0.4950924217700958], [-0.19647853076457977, -0.3633872866630554, -1.1526532173156738, -1.2991784811019897, 0.611031174659729, -0.24881227314472198, -1.024230718612671, 0.9924744367599487, -0.08344002813100815, 1.0591119527816772, 0.8276498913764954, 1.0168031454086304, -0.18875110149383545, 0.511099636554718, 1.124118685722351, -0.4479370713233948, 0.03233039751648903], [1.5647342205047607, 0.9537439942359924, -2.096764326095581, 1.2737987041473389, -0.6699539422988892, 0.6731763482093811, -0.15103566646575928, -0.7004371285438538, 0.4849548935890198, 0.13448397815227509, -0.8414009213447571, 0.07830589264631271, 1.8385167121887207, -0.6374372243881226, 0.5571550130844116, -1.095305323600769, 0.0649765357375145], [0.5011369585990906, -1.0025585889816284, -1.0606828927993774, -0.037248313426971436, 0.32822588086128235, -0.9892597794532776, 1.0096508264541626, 0.7711952924728394, 0.5889365673065186, 0.16493827104568481, 1.3625741004943848, -0.4419371485710144, 1.1794159412384033, -0.39362767338752747, -1.7881239652633667, 1.146859884262085, 1.6178553104400635], [-0.2833824157714844, -0.33129557967185974, -1.460297703742981, -1.166874647140503, 0.8556286692619324, -1.2326785326004028, 1.1260838508605957, -0.4111098647117615, -0.4267193377017975, 0.11664845049381256, -0.2319251000881195, 0.32511553168296814, -0.07272008061408997, 1.0672311782836914, 0.08893083781003952, -0.6900253891944885, -0.02483144775032997], [-0.06852339953184128, 1.2663131952285767, 0.9534488916397095, 0.46266311407089233, -0.749071478843689, 0.027765559032559395, -1.4130204916000366, -0.36309295892715454, -1.6861636638641357, -0.45802581310272217, 0.18194684386253357, -1.3679341077804565, -0.3320062756538391, 0.18156978487968445, 0.6534007787704468, 0.3077418804168701, 0.18495561182498932], [0.6529433727264404, 0.3518155515193939, -0.30948400497436523, -0.4264002740383148, 0.1411927491426468, -0.13369090855121613, -0.40303322672843933, 0.33575811982154846, 0.4546896517276764, -1.1689512729644775, 0.14416038990020752, 0.12138710916042328, -0.602683961391449, 1.1488165855407715, -0.26961737871170044, -0.1999911218881607, -0.8958104848861694], [-0.7653914093971252, -1.1253796815872192, 0.5094677209854126, -0.35099661350250244, -0.8276497721672058, -0.9514607191085815, -0.7999479174613953, -0.043896742165088654, 0.24693718552589417, 1.301189661026001, 1.8490718603134155, 0.6832816004753113, -0.5144678950309753, -0.5470446944236755, 0.29698073863983154, -1.2797200679779053, 1.087038516998291], [1.611742377281189, -0.6922876834869385, -1.7548891305923462, 0.8004387021064758, -0.9496439695358276, -0.49468687176704407, 1.8530964851379395, -1.7819137573242188, -1.0156627893447876, 1.3217761516571045, -0.3491780459880829, 2.9089839458465576, 1.3114413022994995, -0.07727072387933731, -0.6365941166877747, -0.8508391976356506, 0.6300454139709473], [0.1827082633972168, -1.510056734085083, 0.6081151366233826, -0.1730954796075821, 0.8177398443222046, 0.3254350423812866, 0.03608435392379761, -0.7739161849021912, 1.0095070600509644, -1.3682655096054077, -1.0159755945205688, 0.17868363857269287, 2.1661243438720703, -1.9482773542404175, -0.5496782660484314, -0.6725025773048401, -0.4534953832626343], [0.009904629550874233, -0.11154017597436905, 0.6350662708282471, -0.4684204161167145, -1.2328810691833496, -0.2232467085123062, 1.3446638584136963, -0.6477347016334534, 0.4130544364452362, 0.7479327917098999, -1.5186535120010376, -1.229844093322754, -0.4294969439506531, -0.566007137298584, 0.8228381276130676, -1.1884480714797974, 1.2496405839920044], [-1.2826342582702637, 0.6596322059631348, -0.15288877487182617, 0.1422256976366043, -1.1480236053466797, -1.5731443166732788, 0.6065098643302917, 0.3431614637374878, 0.9569295644760132, -0.4784448444843292, 0.3402673602104187, -1.572833776473999, 0.018153736367821693, 0.5940036177635193, -0.049041498452425, 0.5342503786087036, 0.32173076272010803], [2.090549945831299, -0.7247042655944824, 1.196485161781311, 0.4776245653629303, 0.9788965582847595, -0.19547325372695923, -0.2885722219944, -0.8359050154685974, 0.5480294227600098, 0.37293487787246704, 1.3476921319961548, 0.18619228899478912, -0.6892811059951782, 0.6206260323524475, -1.577691912651062, -0.4412510097026825, 0.7815706133842468], [-0.1245347410440445, -0.9972959756851196, -0.4336980879306793, -1.3866063356399536, -0.9245699048042297, 0.27375611662864685, -0.45849886536598206, 0.11363241076469421, -0.22514158487319946, 1.7413060665130615, 0.8799510598182678, -0.012190030887722969, 0.7817676067352295, -0.1544123739004135, 1.3104350566864014, -1.6944400072097778, -1.1294224262237549], [-1.0920400619506836, -0.6160244345664978, 0.5657579302787781, -1.191408395767212, 0.31843051314353943, -1.1964943408966064, 0.6025179624557495, 0.22525344789028168, 2.3604681491851807, 0.27821025252342224, 0.25010350346565247, -0.3561581075191498, 0.6166444420814514, -0.6077175736427307, 0.2196899801492691, 1.6669186353683472, -0.16282454133033752], [0.8439173102378845, -0.21303676068782806, -1.9244792461395264, 1.4299635887145996, 0.8612971305847168, 0.5571126937866211, -0.7156461477279663, -1.1890827417373657, -1.234380841255188, -1.553139328956604, 0.030859220772981644, -1.2112106084823608, -1.2993714809417725, -0.4337955117225647, -1.7863833904266357, 0.3733150362968445, 0.8763703107833862], [-0.4742545783519745, 1.4702407121658325, 2.396798610687256, -0.4689350724220276, 1.4712637662887573, 1.1844055652618408, 1.0921423435211182, -0.9894455671310425, 0.22142325341701508, -1.1531039476394653, 1.474717140197754, -0.8679397106170654, 0.35237449407577515, 0.5883228182792664, 0.5233875513076782, 0.18333250284194946, 1.6825636625289917], [1.3808465003967285, -0.8089465498924255, 0.18520107865333557, 0.09011763334274292, 1.3956407308578491, -0.8380396366119385, -0.519393801689148, -0.43114566802978516, -0.28231576085090637, 1.5884565114974976, -2.076979160308838, 0.5863052606582642, 0.566376268863678, 1.0058128833770752, -2.149325370788574, 1.5094965696334839, 1.5791822671890259], [-0.3151686191558838, 0.18195459246635437, -1.8921111822128296, 0.04643348231911659, 0.13121315836906433, -1.3179713487625122, -0.07646290212869644, -0.18700271844863892, 0.1634671837091446, -0.5310405492782593, 0.8371116518974304, 1.5032014846801758, -3.0510826110839844, 0.3426210284233093, 0.5129513740539551, -1.5825444459915161, 0.18834447860717773], [-0.0388273261487484, -0.22443905472755432, 0.4980889558792114, -0.5362154245376587, -0.05399991199374199, 2.0281167030334473, -1.1247367858886719, 1.1766326427459717, 0.7143967747688293, -0.6318087577819824, 1.3930984735488892, -1.9454259872436523, 0.4052238166332245, 1.8483508825302124, 0.06392647325992584, 0.49437087774276733, -0.3344608545303345], [-0.8440849184989929, -0.5824710726737976, -0.5705906748771667, -0.20900094509124756, -1.7883292436599731, -0.49872255325317383, -1.1771070957183838, 0.8879883885383606, 1.7577682733535767, -0.6145899295806885, -0.7418619394302368, 0.8081610202789307, 0.5365762710571289, -0.047706007957458496, -0.781420886516571, 0.8054764270782471, 1.12943434715271], [-0.23970502614974976, 1.551622986793518, 0.09425012767314911, -0.17918093502521515, 1.2366485595703125, -0.830584704875946, -0.35814908146858215, -1.0198757648468018, 1.1532347202301025, -3.6820106506347656, -0.8452349305152893, 0.7876571416854858, -1.098248839378357, -1.3300129175186157, -2.1061413288116455, 0.7636364698410034, 0.7966006994247437], [0.012012303806841373, -1.422052264213562, -1.4985605478286743, 1.1193749904632568, -1.3877813816070557, -0.7421177625656128, -1.7415391206741333, 0.8631049990653992, 1.1557501554489136, -0.5467142462730408, 1.6250170469284058, -0.01204072218388319, 0.5383591651916504, -0.4934552013874054, -0.5010583996772766, -0.5905324816703796, 0.7226392030715942], [-1.2533440589904785, 0.8323655724525452, 0.9548309445381165, -1.617540955543518, 0.8994558453559875, 1.3789870738983154, 2.1227071285247803, 0.5015844702720642, -1.5247197151184082, -0.2175070196390152, -0.39871981739997864, 0.163655623793602, -0.7325596213340759, -0.45887377858161926, 0.7777352929115295, -0.6435869336128235, 0.23219209909439087], [0.6081719398498535, -1.4467957019805908, -0.39576274156570435, -0.08195918053388596, -0.38789626955986023, 1.370469570159912, -0.002387754386290908, 1.8427379131317139, 0.6726021766662598, -0.29241085052490234, 0.8332924246788025, 0.34936583042144775, 1.3025606870651245, 0.028577357530593872, 0.2625999450683594, 0.2627072036266327, 2.264882802963257], [1.483067274093628, -0.16546176373958588, 0.8450397253036499, 0.3457656502723694, -1.3091577291488647, -0.7843402624130249, 0.7732645869255066, -1.260877251625061, 0.7594587802886963, -0.4917204976081848, -0.2310536503791809, -1.0550155639648438, -0.28608474135398865, -0.38580939173698425, -1.2899088859558105, 0.018026605248451233, 1.1239851713180542], [-1.148285150527954, 1.7528899908065796, -0.041488852351903915, -0.36865925788879395, 0.022777268663048744, -0.4923316538333893, 0.21989521384239197, 0.1754438579082489, 1.4672706127166748, 0.6094395518302917, -0.9587857127189636, -0.8885427713394165, 0.9434843063354492, 1.4019371271133423, 2.7442493438720703, 1.978119969367981, 1.173797607421875], [-0.7771890759468079, -0.848716676235199, 0.6875439882278442, -1.1381548643112183, 0.4456825852394104, 0.8163893818855286, 0.23701420426368713, -0.5741493701934814, 1.6321167945861816, 1.3356236219406128, 0.6619048118591309, 0.36392101645469666, 2.4677414894104004, 0.5345583558082581, -0.553932249546051, 0.4987495541572571, 0.348403662443161], [-0.4548204243183136, 1.4846916198730469, 1.9872864484786987, 1.301128625869751, 0.7744258642196655, 0.6402292847633362, -1.223157525062561, -0.6188077926635742, 1.535636305809021, -3.856645107269287, 1.9701522588729858, 0.567660391330719, -0.19951573014259338, 1.0333980321884155, -0.6661381125450134, -1.6253018379211426, 1.7019375562667847], [0.08488062024116516, -0.22672151029109955, -1.8375146389007568, -1.369370937347412, 1.4708502292633057, 0.46790558099746704, -0.956646203994751, -1.6222808361053467, 1.1923513412475586, -0.6086016893386841, -0.18214043974876404, 0.9965210556983948, 1.1873952150344849, 1.2385112047195435, 0.01719375140964985, 0.6234580874443054, 0.9290858507156372], [0.2985297739505768, -0.708886981010437, -0.042611751705408096, 1.6616777181625366, 0.6388309597969055, -1.0902982950210571, -0.04724613577127457, -0.3611423075199127, 0.8828582167625427, 0.8851910829544067, 2.1258649826049805, -1.8518199920654297, 0.7705463171005249, -0.5742558836936951, 0.7727624177932739, -0.13738000392913818, 1.254118800163269], [1.376200556755066, 0.6808976531028748, -1.087694764137268, 0.45715802907943726, -1.0386440753936768, -0.6614872813224792, 0.4317258298397064, -0.6739415526390076, -1.440979242324829, -0.6020785570144653, -0.9134638905525208, -1.0119518041610718, 0.6269599795341492, 1.1687064170837402, -0.5674886703491211, 0.7652180194854736, 2.157756805419922], [1.4072552919387817, -0.8416486978530884, -1.6108348369598389, 0.23587264120578766, -0.6699694395065308, -0.8565428853034973, 0.5057812929153442, 0.8287559151649475, -0.3213944137096405, -0.16707825660705566, -1.603739857673645, 0.9829102158546448, 0.9057615995407104, 0.9018261432647705, -0.012565495446324348, -0.28187471628189087, 0.397174209356308], [-1.8460919857025146, 0.31682541966438293, -1.234593152999878, -1.313987374305725, 1.5461772680282593, -0.8142108917236328, -0.20393279194831848, 0.5160619020462036, -0.42987871170043945, -1.2596651315689087, -0.16065502166748047, 1.3286614418029785, 1.3075915575027466, 0.08436655253171921, 1.4836161136627197, -1.7691588401794434, 0.06598549336194992], [-1.556105136871338, -1.3864165544509888, -0.7759038805961609, -0.5417742133140564, -1.7078791856765747, -0.34103310108184814, -0.6365655064582825, -0.6158847808837891, 0.35624223947525024, 1.4008773565292358, -0.7040477395057678, -0.5017042756080627, 1.4172276258468628, -0.40178412199020386, 1.8445324897766113, -2.135883331298828, -0.11113587021827698], [-1.9749780893325806, -0.25110000371932983, 0.3267953097820282, -2.0777783393859863, 0.34538981318473816, 0.749322772026062, -0.5166470408439636, 1.2335869073867798, 1.5098645687103271, 0.5899016857147217, -0.2442055493593216, -0.49125730991363525, 0.17104962468147278, 0.08984051644802094, 2.2121737003326416, 0.7296258211135864, 0.9896727204322815], [-0.1558043360710144, 0.5981718897819519, 1.3151623010635376, -1.525619387626648, -0.6963227987289429, -0.3786168694496155, 1.1941580772399902, 0.5702400803565979, -1.040116548538208, 0.4122526943683624, -0.4042187035083771, 0.21929338574409485, 0.5048091411590576, 1.136755108833313, 0.2144542634487152, -0.18489867448806763, 1.8151869773864746], [-0.12936806678771973, 0.60270094871521, -1.0971031188964844, 0.06803286820650101, 1.1427587270736694, 0.12420900166034698, -0.5814512372016907, 0.6548081636428833, 0.4631615877151489, 0.4162291884422302, 0.24286706745624542, 0.23597876727581024, -1.1881076097488403, 1.9139676094055176, 1.6518943309783936, 0.47939440608024597, -0.3042479157447815], [-0.43491241335868835, 0.17189183831214905, 0.4613499641418457, -0.8941248655319214, -0.14231356978416443, -1.077826976776123, -0.3847334682941437, -0.2471572458744049, -0.48351648449897766, 0.1301298588514328, -1.0440176725387573, 1.3190267086029053, 0.5164405107498169, 1.166504979133606, -0.26764675974845886, -0.11800185590982437, 1.221547245979309], [2.0769002437591553, -0.3310720920562744, 0.2665882110595703, -3.1574790477752686, 0.6401662826538086, 0.539240300655365, 2.358872413635254, 0.21977682411670685, -0.01593419723212719, -0.8966653347015381, 0.995475709438324, 2.5781702995300293, -0.008755660615861416, -1.3216638565063477, -1.450398325920105, 3.0196642875671387, 0.6219026446342468], [0.7614924311637878, -0.48977014422416687, 0.43266671895980835, 1.7001386880874634, 0.1636534482240677, -1.5668865442276, 0.4669532775878906, 1.1052252054214478, -0.8895825147628784, -0.8118402361869812, -1.2722896337509155, 0.19637827575206757, -0.6697951555252075, -0.44740304350852966, -0.05450566112995148, -1.8063781261444092, -0.9501050114631653], [0.7337865233421326, 1.572174072265625, 0.33988189697265625, 1.7879185676574707, -0.007172473706305027, 0.2643416225910187, -0.601477861404419, 0.7755031585693359, 0.45025819540023804, 0.3609442710876465, -1.092371940612793, -1.34555184841156, 0.8509366512298584, 0.6214695572853088, 1.5871607065200806, 0.39135509729385376, 1.3027187585830688], [-1.3146671056747437, -1.5059044361114502, 0.12592630088329315, 1.0777000188827515, 0.484433650970459, -0.35960984230041504, 0.10013952851295471, -0.46812689304351807, -1.3395397663116455, -0.27512767910957336, 0.5140822529792786, -0.6584889888763428, -0.6777388453483582, -0.8573706150054932, 0.9499140381813049, -0.674934983253479, -0.799740195274353], [-0.09984443336725235, 0.2514756917953491, -0.5805671215057373, -0.7658705711364746, -0.733373761177063, -0.6768840551376343, 1.371695876121521, -0.3465884327888489, -0.7348087430000305, -3.4456899166107178, 0.08260618895292282, 0.10621914267539978, 1.4603607654571533, 1.2978955507278442, 0.9439421892166138, -0.7927992343902588, 0.837784469127655], [0.4516921043395996, -0.1727665662765503, 1.0927790403366089, -1.2300890684127808, -0.39824560284614563, -0.000494533043820411, -0.6207035779953003, 2.1018869876861572, -0.8507052659988403, -0.5242946743965149, 0.7471786737442017, 1.1702805757522583, -1.0291656255722046, -0.1625579595565796, -0.034256935119628906, 1.5858657360076904, -1.1612548828125], [-2.425565719604492, 0.06425104290246964, 0.23021312057971954, 0.19840958714485168, 0.25803762674331665, -1.1708166599273682, 2.1205241680145264, -0.5520254969596863, 0.7481460571289062, -1.3644040822982788, 0.878010094165802, -0.7821663618087769, -0.3403812348842621, 0.08922470360994339, 1.0129644870758057, -0.5611487627029419, 0.22924485802650452], [0.08704905956983566, 0.3942355811595917, -1.8440872430801392, 0.3148054778575897, -1.6418787240982056, 1.5647430419921875, -0.4360114634037018, -1.3620741367340088, -0.5483623743057251, 0.17718978226184845, -0.21347475051879883, 0.12580782175064087, 0.12448970228433609, 0.6719600558280945, -0.058188654482364655, 1.3578686714172363, 0.36416420340538025], [0.27847129106521606, 1.1379618644714355, 0.24119511246681213, 0.8200883865356445, -0.4655836522579193, 0.09072385728359222, 1.0814344882965088, -0.006351031828671694, 1.2896983623504639, -0.5251790285110474, 0.6417970657348633, -1.5567779541015625, -0.10679083317518234, -0.30050128698349, -0.49493634700775146, -0.11932572722434998, -0.6974145770072937], [0.4660795032978058, 0.8321062326431274, 0.39439788460731506, -0.7082009315490723, 0.9267391562461853, 0.502061128616333, -1.0456606149673462, 1.0816686153411865, 0.16573932766914368, -1.5476628541946411, -0.9947661757469177, 0.22809919714927673, -0.6094288229942322, -0.2391103357076645, 0.9191532731056213, 0.07760097086429596, 1.3469163179397583], [1.0841401815414429, -0.3114015758037567, 1.0649018287658691, -2.3669981956481934, 0.2538934648036957, 0.33831486105918884, -0.0007428752724081278, 1.0625810623168945, -1.092038631439209, -1.9031164646148682, 0.107280433177948, -0.09611888229846954, -1.1978034973144531, -1.7676223516464233, 2.4394094944000244, -0.5516791939735413, 0.6900217533111572], [1.5457056760787964, 1.6700599193572998, -3.2434725761413574, -1.85053288936615, -0.13562598824501038, 3.173544406890869, 0.31347545981407166, 0.2689836621284485, 0.8417083621025085, -2.4418680667877197, 0.12872131168842316, -2.8996593952178955, 1.0238745212554932, 1.7473433017730713, -0.02802617847919464, -0.8308216333389282, -0.3401339054107666], [-0.0955081656575203, 0.2568936049938202, -0.2460419088602066, 3.8104052543640137, -0.7247316837310791, 0.6238469481468201, 1.0454716682434082, 0.7015715837478638, 1.1122565269470215, 0.019331280142068863, 0.6859053373336792, -0.12899360060691833, 0.25839948654174805, -0.5247901678085327, -1.7492140531539917, -2.974895715713501, 1.0945658683776855], [0.7170608639717102, -0.023754840716719627, -1.189743995666504, 1.8549209833145142, -0.8900614380836487, 1.325002670288086, 1.082912564277649, -0.8439061641693115, 0.4416763186454773, -0.08711238205432892, -0.3939167857170105, 0.49191588163375854, 0.6561587452888489, -0.5744782090187073, -1.3233447074890137, -1.822944164276123, -0.11655738949775696], [-0.2809992730617523, 0.56302410364151, 0.7930020093917847, -0.20395639538764954, 0.5324060916900635, -0.9891524910926819, 1.2534348964691162, 1.0098005533218384, -1.0546995401382446, -0.1388288289308548, -1.557690143585205, -0.8841038942337036, 1.538293719291687, -0.9551968574523926, 0.3408409059047699, 0.6466360092163086, -0.032500576227903366], [1.3699268102645874, -1.487392783164978, -0.7120211720466614, 3.2315781116485596, 0.8538769483566284, 0.27488434314727783, 0.5483004450798035, -0.12110182642936707, 1.3767547607421875, -0.18836656212806702, 2.6070923805236816, -2.340999126434326, 1.147398591041565, 0.3560416102409363, 0.16657854616641998, 0.8427804708480835, -2.317786931991577], [1.3343585729599, 1.3527835607528687, -0.6706606149673462, -0.8337714672088623, -0.026597220450639725, 1.0934034585952759, -0.27441373467445374, -1.0114638805389404, 0.5992712378501892, 1.3772658109664917, -0.2391812950372696, 0.553495466709137, -0.938249409198761, -0.1441827416419983, -0.982262909412384, 1.2910677194595337, 0.8678421378135681], [-2.0152804851531982, -0.15133479237556458, -1.0045033693313599, -0.23944218456745148, -0.4184788465499878, 0.9208685159683228, 1.1745057106018066, 0.7623217701911926, 1.960351586341858, -0.3912416696548462, 0.4179367423057556, -0.6188685297966003, -0.5341367125511169, 0.04442436993122101, 0.3906605541706085, 0.17624998092651367, -0.7639389038085938], [0.8377082347869873, -0.003227807814255357, -0.23128163814544678, 0.9435101747512817, 0.1702912598848343, 0.4401121437549591, 0.8068438172340393, -1.7161753177642822, -0.5743919014930725, -1.3179705142974854, -1.007075548171997, 1.570027232170105, 1.3446639776229858, -0.6084546446800232, 1.2043051719665527, -1.1980537176132202, 1.2712658643722534], [0.18363606929779053, 0.8511177897453308, 0.701653778553009, -1.2056164741516113, -0.21439525485038757, 0.24420569837093353, -0.36862829327583313, 1.2339717149734497, -1.3543978929519653, -0.9430087208747864, -0.8358013033866882, -1.3849928379058838, -2.200401782989502, -0.5157629251480103, -1.2755039930343628, -0.3091389238834381, 0.686212956905365], [0.04541884362697601, -0.8238169550895691, -0.1389600932598114, 0.547674298286438, 0.6468896865844727, 0.9227767586708069, 1.3889570236206055, 0.7593052983283997, -0.007647140417248011, -0.3810760974884033, 0.2564181685447693, 1.4876114130020142, -0.35070210695266724, 1.5400726795196533, -0.7103859782218933, -1.1932896375656128, 0.45574939250946045], [-0.18102960288524628, 0.7610654830932617, 1.2603131532669067, -0.61114102602005, 1.328173041343689, -0.32365891337394714, 0.8921738862991333, 1.161761999130249, 0.4218868017196655, 1.347875714302063, 0.2462644726037979, -0.7780516147613525, -0.8107067346572876, -0.08136896789073944, -0.825554609298706, -0.018054835498332977, 1.3822910785675049], [-0.8659159541130066, 0.3908484876155853, -0.40818384289741516, -0.80417400598526, -1.0275614261627197, -0.7933369874954224, 0.49483609199523926, -0.1116122454404831, 1.6233776807785034, 0.5522421598434448, 1.091407299041748, -2.296452283859253, -0.922211229801178, -0.9379850625991821, -0.40342947840690613, 0.5482645034790039, -2.565565824508667], [0.13378995656967163, -0.7709450125694275, -1.4217253923416138, 0.6132960319519043, 0.572669506072998, 1.0148459672927856, 1.2098445892333984, -0.520626962184906, -1.0796321630477905, 0.563199520111084, 0.14337557554244995, 0.5284461379051208, 0.393129825592041, 0.41962912678718567, -0.10965725779533386, -0.8081720471382141, -0.5955333709716797], [-0.7438592910766602, -1.0478605031967163, 1.5341277122497559, -0.5492784976959229, -0.8248872756958008, -0.4622684717178345, 0.9084652662277222, 0.8979455232620239, -0.153204083442688, 0.6253681778907776, 0.4175253212451935, 0.5846940279006958, 0.12586325407028198, 0.354458212852478, -0.31229615211486816, 1.0077546834945679, -0.11631845682859421], [0.36069440841674805, -0.27181702852249146, 0.28038257360458374, 0.2864074409008026, 0.8970882892608643, -0.42903023958206177, -0.09945129603147507, 0.6047181487083435, -1.1907020807266235, 0.35163453221321106, 2.194599151611328, -0.6519295573234558, 1.131386399269104, -0.4453047811985016, -0.848305344581604, -0.8311576843261719, -0.26174265146255493], [-1.3943880796432495, 0.9182863235473633, -0.05749576911330223, -0.06292636692523956, 1.185549020767212, -0.18092019855976105, -0.29482167959213257, 1.3982847929000854, -0.10107720643281937, -1.8922393321990967, 0.06392528116703033, 1.677469253540039, 0.0031221224926412106, -0.6675858497619629, 1.940720796585083, -0.010635016486048698, -0.3719491958618164], [0.16587065160274506, -1.6867698431015015, -0.2543833255767822, -0.1566975712776184, -1.7495903968811035, 1.5999809503555298, -1.385402798652649, 1.4878768920898438, 1.0497618913650513, 0.34168192744255066, 1.480432152748108, 2.334855556488037, 0.3027384877204895, -0.3016403913497925, -1.3931267261505127, 0.20945191383361816, -1.1099934577941895], [1.1785922050476074, 2.4656896591186523, -0.8948923945426941, -0.6221250295639038, -0.5733286738395691, -0.5265111327171326, 1.0670019388198853, -1.1819846630096436, -0.08435770869255066, 0.7057043313980103, 0.03662986680865288, -0.7754052877426147, -2.49668550491333, -0.6679185032844543, 0.38174813985824585, 0.33748844265937805, -1.0271518230438232], [-0.08098632842302322, -0.182047501206398, -0.1348603367805481, 0.38581809401512146, 0.014954326674342155, -0.6371369957923889, -1.247755527496338, -0.5669813752174377, 0.37412938475608826, -0.7160884737968445, -0.9111648797988892, 0.2134329378604889, -0.6847049593925476, 0.3575514554977417, -1.3606711626052856, 0.6169224381446838, 1.8139089345932007], [1.2335944175720215, 0.7133345007896423, 0.9659938216209412, -0.8166411519050598, -0.383903831243515, 1.3718103170394897, 0.5562500357627869, 1.080582618713379, 0.9714825749397278, -0.11952835321426392, 0.953473687171936, 0.8578763008117676, 0.31329408288002014, 0.7861100435256958, 0.06605662405490875, -1.067995548248291, -0.09129767119884491], [-0.04243708774447441, -1.530687689781189, 1.5702545642852783, -1.7689545154571533, 0.26131516695022583, 1.0050454139709473, -0.47158852219581604, -2.047853469848633, 1.1697218418121338, -0.5351754426956177, -1.7206499576568604, 0.6053401231765747, 1.1688350439071655, -1.2226977348327637, 0.09739252924919128, -0.7543166279792786, -0.525256872177124], [-0.07323024421930313, -1.3470436334609985, -1.0614930391311646, -0.08495493233203888, -1.0428060293197632, -0.025679731741547585, 2.471977710723877, 1.3723372220993042, 0.17317543923854828, 0.1447637975215912, 1.1492568254470825, 0.07104206830263138, 1.1067613363265991, -2.7787024974823, 1.321325659751892, 0.6208415031433105, 0.38921618461608887], [-1.0258524417877197, -0.5104876756668091, -0.2521185874938965, 0.4347033202648163, -0.4684315621852875, -0.07426067441701889, 1.7446845769882202, 0.5935204029083252, -0.9580885171890259, -0.4556257426738739, -0.2372896671295166, -1.9607189893722534, -2.2912137508392334, 2.5902535915374756, 0.6616678237915039, -0.5496907830238342, -0.1827264428138733], [1.1724367141723633, -0.17080970108509064, -0.04481707140803337, -0.2312077134847641, 1.244544267654419, -0.42414841055870056, 1.2375584840774536, 0.5688925981521606, 0.10365170985460281, -0.09540543705224991, 0.589176595211029, -0.06730121374130249, 1.3227061033248901, -1.826795220375061, -2.010993003845215, 1.291536808013916, 0.6922377347946167], [0.10083946585655212, 0.41267117857933044, 0.15448522567749023, 0.25915148854255676, -1.6875039339065552, -0.9236868619918823, -0.4295292794704437, -0.311587393283844, 0.17487317323684692, 1.0651769638061523, 1.447590708732605, -1.4287313222885132, 0.4848746061325073, -2.0117974281311035, 2.631314754486084, -1.3769043684005737, -0.1888251006603241], [0.5693201422691345, 0.6672455668449402, -1.210353136062622, 0.7553412914276123, 0.27079349756240845, 2.1411890983581543, 1.3881512880325317, 0.6251542568206787, -1.1154356002807617, -0.4428400695323944, -0.989092230796814, 0.7188749313354492, 0.8874117732048035, 0.7229117155075073, -0.3222511112689972, 0.7348690629005432, -0.2690679728984833], [0.9342400431632996, 1.544221043586731, 0.4252537786960602, 0.6666447520256042, 1.8954137563705444, -0.1298762857913971, -0.5179076194763184, 0.09529417753219604, -0.10189560800790787, 0.4814161956310272, -0.28367406129837036, -0.2364388406276703, 0.5435380935668945, -0.6570900678634644, 1.4614183902740479, -2.1581931114196777, 1.0253145694732666], [0.8794940710067749, -0.6425522565841675, -0.3439176082611084, 0.44603729248046875, -1.005654215812683, -1.84394109249115, -1.279883623123169, -0.9357266426086426, 0.49789854884147644, -1.0608054399490356, 1.0419259071350098, -0.5191189050674438, 0.5754461884498596, -1.616637945175171, 0.7508535385131836, -0.5800204277038574, 0.5047019720077515], [0.8257828950881958, 1.0409200191497803, 0.6797121167182922, -0.18497465550899506, 0.20418322086334229, 1.0191679000854492, -0.29515692591667175, 0.4143260717391968, -0.26377397775650024, 1.9871244430541992, -1.2148209810256958, 0.793453574180603, 1.875974178314209, -0.18073274195194244, -0.7874460816383362, 1.145665168762207, 1.3983699083328247], [0.4421493709087372, -0.5792100429534912, 0.10839501768350601, -0.4207558035850525, -2.1212403774261475, 0.9971691370010376, 0.5971315503120422, -0.8021324872970581, 0.817683756351471, -0.26947858929634094, 1.7176648378372192, -1.503675103187561, -0.5255647301673889, 0.21023054420948029, -1.0358737707138062, -0.8759387135505676, 1.2074047327041626], [0.47819647192955017, 0.7738398909568787, -1.246389627456665, 0.16364550590515137, -0.5700950026512146, 2.797192335128784, 0.6474305391311646, -0.9033392071723938, -0.6517125964164734, 0.29543623328208923, 0.08647073805332184, 0.46910667419433594, 1.880919337272644, 1.8170017004013062, -0.6620450019836426, -0.8626857995986938, 0.8625379800796509], [-0.39137881994247437, -1.7221759557724, -0.07627779990434647, 0.8372460007667542, -0.37624430656433105, 0.2548834979534149, 0.4451260566711426, 0.5911588668823242, 0.19147668778896332, 0.375276654958725, -0.8703665137290955, 0.3121529519557953, 0.8943037390708923, 0.40658751130104065, -1.161655068397522, -0.8532681465148926, -0.9711018204689026], [-1.459682822227478, -1.3142051696777344, -1.3962981700897217, -0.8388282060623169, 2.0203616619110107, 0.344514399766922, 1.2792518138885498, 0.3770817816257477, 0.09845045953989029, -0.2366170436143875, -1.237650752067566, 0.49782904982566833, 0.7751243114471436, 0.4377211928367615, -0.059686485677957535, 0.7560631632804871, -1.850376009941101], [0.3010201156139374, -1.8056772947311401, -1.0119476318359375, 0.3796243965625763, 1.0701242685317993, 1.0385611057281494, 1.7406195402145386, -0.6063562631607056, -0.04474056512117386, 0.052774395793676376, -0.2884635031223297, 1.6755526065826416, -0.6262791752815247, 3.0760695934295654, 1.656455159187317, 0.6180282831192017, -0.8690791130065918], [-0.7922554016113281, 0.6109910607337952, 1.2219587564468384, 0.463300883769989, -1.6405388116836548, -0.39109107851982117, 0.5009892582893372, 0.04295898973941803, 1.336484432220459, -0.35096266865730286, 0.19501346349716187, -1.75923752784729, -0.5480561852455139, 0.1491067111492157, 0.44929254055023193, 1.0725023746490479, 0.9625846743583679], [0.2963757812976837, 0.3415758013725281, 1.1207376718521118, 0.1741448938846588, 0.2859503924846649, 1.119422435760498, 1.0541162490844727, -1.7339589595794678, 0.8109431862831116, -0.8265237212181091, -0.562759280204773, 1.373259425163269, -0.6722038388252258, 0.6066848635673523, -1.4752418994903564, 0.11047732830047607, -1.5079818964004517], [-0.058160651475191116, 1.4575858116149902, 0.3557122051715851, 0.22433042526245117, 2.3071084022521973, -1.4587467908859253, 0.2049701064825058, -0.5514456033706665, 0.47589364647865295, -1.1254255771636963, -1.5266621112823486, 0.8766340613365173, 1.107108473777771, -0.8698161840438843, 0.6597860455513, -1.364490032196045, 1.1969093084335327], [-0.8316385746002197, -0.5890463590621948, -0.5994176268577576, -1.692394495010376, 2.1183817386627197, -0.1731935292482376, 0.08939941972494125, 0.029726777225732803, -0.13424800336360931, -0.5258964896202087, 1.532248616218567, -0.5893707871437073, 0.14988738298416138, -0.31304168701171875, 0.16532640159130096, 0.5144763588905334, 0.5393975973129272], [-1.0469428300857544, 1.177193522453308, -0.02643914520740509, 1.2594881057739258, 0.9549153447151184, -2.5537307262420654, 1.4071904420852661, -0.4467625319957733, -0.9346739053726196, -0.15661334991455078, -1.4663928747177124, -0.0744357481598854, 0.9171956777572632, 0.9008623957633972, 0.18208001554012299, 0.70615553855896, -1.162848711013794], [-1.0339254140853882, 0.36952346563339233, -2.492511034011841, 0.9890034794807434, -0.08604603260755539, -0.6249339580535889, -0.6476057767868042, 1.363869309425354, -2.0046913623809814, -1.3755947351455688, 1.5854946374893188, -0.3707066774368286, 0.06119624152779579, -0.21101391315460205, 0.6820775866508484, -0.6522862315177917, 1.0338002443313599], [-0.890379011631012, 0.3096123933792114, 2.1798722743988037, 0.5588898658752441, 2.293717384338379, -0.9393575191497803, -0.24190472066402435, 0.30216264724731445, -2.020933151245117, 0.9477089047431946, -0.6585546731948853, -0.026356760412454605, -0.6739440560340881, -0.9007466435432434, 0.550661563873291, 0.4442494213581085, 1.1240094900131226], [-0.18691964447498322, -0.21155962347984314, -0.5974018573760986, -0.15030093491077423, 0.6959460973739624, -0.2020859271287918, 1.5102283954620361, 2.400078296661377, 0.061080146580934525, 0.3506990373134613, 1.1947200298309326, 0.20000162720680237, 0.39295998215675354, 0.04548189789056778, -1.2261477708816528, -0.44836628437042236, -2.252715587615967], [0.3220837712287903, -0.6629322171211243, -0.7202959060668945, 0.34312713146209717, 0.1294860988855362, -0.903845489025116, -0.4275168776512146, 1.3862735033035278, -0.8647121787071228, -1.496058702468872, 0.2674334943294525, 0.024765992537140846, -0.14434777200222015, 1.2500697374343872, 0.4199051856994629, 0.23753948509693146, 0.8455886840820312], [0.07708115130662918, -0.31643104553222656, -0.445395827293396, 0.09047994762659073, -2.2657461166381836, 1.3088699579238892, 1.0292887687683105, -1.2070286273956299, -0.1746043860912323, -1.0736418962478638, -0.6440585255622864, -0.8380193114280701, 1.322385311126709, 1.1282109022140503, 0.21348977088928223, -2.2580976486206055, 1.151313304901123], [0.04975799843668938, 0.4619026780128479, 1.0817362070083618, 0.5103049278259277, 0.4069697856903076, -0.7536923885345459, -1.6591209173202515, 0.8856450319290161, -0.06106392294168472, 0.7842028141021729, -0.45285576581954956, 0.468205988407135, 0.6776341199874878, 0.59920734167099, 0.046224262565374374, 0.8272595405578613, 0.3699941337108612], [-0.04518831521272659, -0.24827277660369873, 0.8168962001800537, -0.26293981075286865, 0.48586636781692505, -1.1933817863464355, 0.03023894876241684, -0.07755978405475616, -0.1507442593574524, -1.1200647354125977, 1.4891365766525269, -0.36435040831565857, 0.5025010108947754, 0.7393044829368591, -1.285814881324768, 0.9694972634315491, 0.933079719543457], [-1.0849881172180176, 1.0289109945297241, 0.17162847518920898, -0.639051616191864, -0.17677339911460876, -1.235147476196289, -0.013974119909107685, -1.8215487003326416, 2.1565146446228027, 0.1360214501619339, 0.7124761939048767, 0.3398248553276062, 1.489369511604309, 1.3742854595184326, -0.5586833953857422, 0.8479833006858826, 0.2984636723995209], [-0.7401323318481445, -0.6583539843559265, -1.1328527927398682, 1.1269603967666626, -0.028056729584932327, -0.10107404738664627, 0.217030331492424, 1.3037718534469604, -0.6119663715362549, 1.7320228815078735, 0.3276302218437195, 0.05878487974405289, 0.7285025715827942, 0.5685154795646667, 0.6207790374755859, 1.3082101345062256, 1.8764019012451172], [0.3723461627960205, 0.5227602124214172, -0.637724757194519, 1.9257698059082031, 1.4678165912628174, -1.0376017093658447, 0.16688306629657745, 0.4400746822357178, -0.06701914966106415, 1.4234280586242676, -1.197652816772461, -0.6158742308616638, -0.11254256963729858, 1.8504692316055298, 0.5121481418609619, 1.8393563032150269, -0.31323492527008057], [0.5388131141662598, 0.17711782455444336, -1.2901047468185425, -0.28454267978668213, -0.6141527891159058, -0.8812637329101562, 0.266720175743103, 0.23568414151668549, 0.6234188675880432, 0.3230985999107361, -0.04741405323147774, 0.16560271382331848, 0.293565034866333, -0.041395921260118484, 0.10860995948314667, 0.28788116574287415, -1.368371605873108], [0.33743715286254883, 0.7713090777397156, 0.06231595575809479, -0.03253627195954323, -0.3183160722255707, 1.1088910102844238, -0.526088535785675, -1.8461167812347412, 0.7807695865631104, -1.6178191900253296, -0.8790240287780762, 0.5279943943023682, 0.9562826156616211, 0.9467024803161621, -2.4258534908294678, 1.0074034929275513, 0.31420841813087463], [0.9734377861022949, 2.0201051235198975, 1.0650362968444824, 1.5747318267822266, 0.017917154356837273, -0.1372431516647339, -0.5252816677093506, -1.3289941549301147, 0.46948713064193726, -0.06058100238442421, -0.6859727501869202, -1.219483733177185, 1.1043550968170166, -0.36029255390167236, 0.23246367275714874, -0.6259264945983887, 0.644920825958252], [-0.5154722929000854, -0.19532258808612823, 2.071357250213623, 0.32198724150657654, -1.4088927507400513, 1.7127563953399658, -2.069262742996216, 0.345519095659256, 1.1783795356750488, -0.048711419105529785, -0.21900469064712524, -0.3028828501701355, 0.08593636751174927, -0.07914643734693527, 2.9943251609802246, 1.8183343410491943, 0.764316976070404], [-0.7085056900978088, -0.16183209419250488, -0.46153756976127625, -0.11437126994132996, -1.1999403238296509, 0.32863500714302063, 0.8719640970230103, -0.5069530606269836, 1.271291732788086, 0.19990430772304535, -0.9350630044937134, 1.2668657302856445, 1.2868362665176392, -1.3664296865463257, 0.814361572265625, -0.5798352956771851, -0.6293669939041138], [0.7313529849052429, 0.3099304437637329, 0.7845302224159241, -0.11090295761823654, 0.6912229657173157, 1.4709163904190063, 0.49324873089790344, 0.8617206811904907, -0.7573177814483643, -0.2662801146507263, 0.4130604565143585, -0.7050050497055054, 0.8812790513038635, 0.18318043649196625, -0.24912099540233612, -0.6447432637214661, -1.2713096141815186], [-1.1800224781036377, -1.0437792539596558, -0.5583421587944031, -0.4349920451641083, 0.48888203501701355, -2.5207014083862305, -0.27801403403282166, 1.5656836032867432, 0.7919912338256836, 0.3728041648864746, -0.2171037197113037, -1.494299054145813, 0.6934119462966919, 0.08341844379901886, 0.9353508353233337, 0.5673262476921082, -0.6138267517089844], [0.768744707107544, -0.653002917766571, 1.0827126502990723, -1.1967521905899048, 1.0086857080459595, -0.46400606632232666, 0.7450372576713562, 1.1244316101074219, -0.5398260951042175, 0.8791004419326782, 0.032726019620895386, 0.15339352190494537, 0.6605374217033386, -2.212367057800293, -0.8126954436302185, 1.2430931329727173, 0.728289783000946], [1.2302846908569336, -0.0086425906047225, -1.1253688335418701, 0.7049756050109863, 0.25585147738456726, 1.0753896236419678, -0.9599315524101257, 1.3747875690460205, 1.1860800981521606, -0.54979008436203, -1.1509963274002075, -1.456083059310913, 0.07262676954269409, 1.3847150802612305, 0.9317201375961304, 0.057791415601968765, -0.29647570848464966], [-0.9163133502006531, -0.18135057389736176, -0.28710484504699707, 1.3045189380645752, -0.12413258105516434, 0.4872932732105255, -0.4506458342075348, -0.47515857219696045, -1.6016734838485718, -1.3590959310531616, 0.17231546342372894, -1.0380395650863647, 0.2998688220977783, 1.0057880878448486, 1.4433612823486328, -0.1928887665271759, 0.1225365400314331], [-0.076494000852108, -0.20185920596122742, -1.493242859840393, -0.19968149065971375, 1.4333175420761108, 0.758740246295929, -1.095292329788208, -0.30721789598464966, 1.4168652296066284, 1.0189157724380493, -0.24451877176761627, 0.8134825825691223, -0.18502813577651978, -0.4187863767147064, -0.764145016670227, 0.8004633188247681, 1.4005773067474365], [-1.1126389503479004, 0.25456055998802185, 0.9010850191116333, 2.9596450328826904, -0.4280644953250885, -0.060663122683763504, -0.8792313933372498, -1.598774790763855, 2.870976448059082, 0.24003781378269196, -0.5902423858642578, -0.19065843522548676, 0.6926513314247131, -1.2800263166427612, -0.48973849415779114, -1.0741643905639648, -2.2199933528900146], [-0.33015090227127075, -0.9995836019515991, -0.8062580227851868, -0.1161608174443245, 1.9590811729431152, 1.8435121774673462, -0.3927413523197174, 0.05321122333407402, -0.20532779395580292, 0.4453190565109253, -0.6611889600753784, -1.0518349409103394, 1.3625816106796265, -1.2662237882614136, 0.02864777110517025, 1.6009423732757568, 1.6848630905151367], [0.09048491716384888, 0.4686978757381439, -0.16604723036289215, -0.3132384717464447, 1.2998393774032593, -1.878272533416748, 0.4432664215564728, 0.779455304145813, -1.211480975151062, -0.4656941592693329, 1.067327857017517, 0.7196142077445984, 0.2063533514738083, 0.42235150933265686, -0.11025066673755646, -0.45502546429634094, -0.010885114781558514], [-0.0744885578751564, -0.23750941455364227, -1.3281341791152954, 1.3983373641967773, 0.560304582118988, -0.8412917256355286, -1.2136160135269165, -0.39270809292793274, -0.19251468777656555, 1.1393967866897583, 0.6868249773979187, -0.3954468369483948, 0.6658913493156433, 0.37852638959884644, 1.007156491279602, 1.1316235065460205, 1.4516531229019165], [0.31371110677719116, -0.628886878490448, 0.3376922309398651, -0.013472195714712143, 0.5421600341796875, -1.3284912109375, 0.5616688132286072, 0.10723867267370224, -0.6215468645095825, -1.0507656335830688, 0.07736217975616455, -0.0070650591515004635, 0.6312112808227539, 3.104506731033325, -1.3935884237289429, -2.6153461933135986, 0.8140552043914795], [0.2556186616420746, 0.6996331214904785, 0.7928483486175537, 0.4026840627193451, -2.4085137844085693, -0.10919792205095291, 1.1136071681976318, 0.17198027670383453, -1.989642858505249, -0.0586511455476284, 0.23488318920135498, -0.7015814781188965, -1.3364520072937012, 0.5452721118927002, -0.3277652859687805, 0.4471471905708313, -1.1246178150177002], [0.5140686631202698, -1.2189439535140991, -2.1910862922668457, 0.03832552209496498, -0.34462201595306396, 0.2925395667552948, 0.23863542079925537, 0.4980289936065674, 1.873987078666687, -0.11395157873630524, 0.37245994806289673, 1.5797638893127441, 1.067204475402832, -1.2304739952087402, -0.1828630119562149, -1.0841401815414429, -0.6894911527633667], [-0.209360733628273, 0.998586118221283, -0.47180160880088806, 0.34768590331077576, 0.5785829424858093, -2.1163578033447266, 0.6558899283409119, -0.43448057770729065, 1.1429742574691772, -0.8496061563491821, -2.2514054775238037, 0.5455579161643982, 1.1808253526687622, -0.23778747022151947, -2.2073755264282227, 0.23793146014213562, 0.9491466879844666], [0.7475073933601379, 0.7816814184188843, -1.3727620840072632, 2.369541883468628, -1.1537219285964966, -0.9184435606002808, -0.7899746298789978, -0.48463794589042664, 0.6354665160179138, -0.05536245182156563, -1.256156086921692, -0.22925366461277008, 1.4562931060791016, -0.5375244617462158, 0.1743241846561432, -0.8377918004989624, 0.18958400189876556], [-1.4945690631866455, 0.4314972460269928, -0.15847018361091614, 1.3657262325286865, -0.07382255047559738, 0.3977413773536682, 1.021899700164795, -0.1820051372051239, -1.2802650928497314, -0.8440017104148865, -1.3487693071365356, -0.12773557007312775, 0.2541966736316681, 0.2513308525085449, -1.0705273151397705, -1.2717441320419312, 1.511505126953125], [-1.3025567531585693, 0.3790733814239502, 1.097439169883728, 0.35361164808273315, -0.08760298043489456, -1.7708791494369507, -0.5120937824249268, 1.705875039100647, -0.07606416940689087, 0.8956040143966675, -1.7642526626586914, -0.8335732817649841, -0.26107633113861084, 1.7724924087524414, 1.1265097856521606, -0.197029709815979, -0.2507496178150177], [-0.5127102732658386, -0.0649246945977211, 0.8450115919113159, -0.5533447265625, -0.7391757369041443, -0.24053217470645905, 0.13578249514102936, 0.06422920525074005, -0.18346893787384033, -1.518147349357605, 0.4021865129470825, -1.2614266872406006, 1.466135025024414, 0.15292888879776, 0.10759546607732773, -1.2091326713562012, 1.4445586204528809], [1.3340426683425903, -0.5101554989814758, 0.44598397612571716, -0.40831780433654785, -0.7822363972663879, 0.39600858092308044, 0.08788423240184784, -0.41085192561149597, 0.5180496573448181, -0.3263750374317169, -1.7802808284759521, -1.868645429611206, 0.042069580405950546, -0.9854345917701721, -0.8994835615158081, -0.0449068620800972, -0.36476171016693115], [0.29889851808547974, 1.5129486322402954, 0.06807346642017365, -0.901029646396637, -1.1508411169052124, -1.4339553117752075, -1.7441879510879517, -1.1728743314743042, 0.05670744180679321, 1.3269277811050415, -0.7616569399833679, 0.7077774405479431, -0.3588399887084961, -0.25452908873558044, -1.1148872375488281, 0.005130526144057512, -0.28754308819770813], [0.6720859408378601, 0.39962291717529297, 1.028536081314087, -0.7048987746238708, -1.6435776948928833, -0.5674525499343872, -0.220612570643425, -1.0142673254013062, -0.5325925350189209, -1.265769600868225, -0.8804214596748352, 1.1996735334396362, 0.5778776407241821, -1.018718957901001, 2.398178815841675, 2.1791820526123047, -0.3516552746295929], [-1.1622955799102783, 0.40391257405281067, -1.660494327545166, 2.437832832336426, 0.0017986706225201488, -0.21604661643505096, -0.7791157960891724, -0.5507553815841675, 0.3876548111438751, 0.6918113231658936, 1.0867031812667847, 0.4981571137905121, 0.7409349679946899, 0.42138075828552246, 0.7766733765602112, 2.1348209381103516, 2.0713703632354736], [1.088473916053772, 0.4176628291606903, 1.4374910593032837, 1.0892254114151, -0.6427976489067078, 1.4888263940811157, -0.5396323800086975, 0.8367741703987122, 0.7636518478393555, -0.7222700119018555, -0.4795530438423157, -0.22808092832565308, 1.042113184928894, 0.3235657513141632, -1.2427072525024414, -0.20644713938236237, -0.12290917336940765], [0.9899088144302368, 2.8900535106658936, -0.42005059123039246, 1.0657223463058472, -0.9296383857727051, 0.7372695803642273, -1.8338819742202759, 0.676616907119751, 1.7887731790542603, -1.5788488388061523, -0.8684588670730591, 0.838990330696106, -0.4881494343280792, -0.3225913345813751, -0.14540256559848785, -0.40821292996406555, 1.2418489456176758], [0.3007895052433014, -0.18739533424377441, 0.41539913415908813, -1.3804208040237427, 0.3174262046813965, 0.9149969816207886, 0.585844874382019, -0.48389747738838196, -0.575799286365509, 0.021693676710128784, -1.6390544176101685, 0.8309758901596069, -0.5477787852287292, 2.119366407394409, 0.41049811244010925, -0.7803254127502441, -1.0773122310638428], [-0.40323635935783386, -0.7612842917442322, -0.2602320909500122, 0.3778757154941559, 0.3737073242664337, 0.005072908475995064, 0.9970221519470215, 1.6242984533309937, 1.0100762844085693, -0.32346904277801514, -0.4510239362716675, -0.09852615743875504, 0.5238403081893921, 0.4175635576248169, 0.7113988995552063, -0.12642531096935272, -0.5538322925567627], [0.019182076677680016, -0.3374561667442322, 0.6096831560134888, -0.7523453235626221, 0.4675616919994354, -0.7231351137161255, -0.7746970057487488, -1.3094478845596313, 0.7429730892181396, 0.826978325843811, 0.27068570256233215, 0.7112294435501099, -1.1940959692001343, 1.693176507949829, -0.4426025450229645, -0.10049189627170563, -1.3339089155197144], [-1.8978122472763062, 0.18615254759788513, 0.21229912340641022, 0.8511483669281006, 1.622780442237854, -1.3771625757217407, 0.8687001466751099, -1.215712070465088, 0.30023303627967834, 0.44522756338119507, -1.4201465845108032, -0.8145498037338257, 1.18455970287323, -0.5448364615440369, 0.010610018856823444, 0.3529244661331177, 0.3046218454837799], [-0.34760528802871704, 0.4644438326358795, 0.30394333600997925, 0.20863085985183716, 1.9690662622451782, 0.3087661862373352, -0.09252487868070602, 0.005371383856981993, -0.4533909261226654, 0.3026788830757141, 0.3952430188655853, 0.09508753567934036, -0.6468134522438049, -0.7354711294174194, -0.13400447368621826, 1.215415120124817, -1.8747670650482178], [-0.4721805453300476, -0.4126077890396118, -0.8604568839073181, 0.36806195974349976, -0.16360607743263245, -0.32378697395324707, -0.33557531237602234, -0.3589426577091217, -1.5763410329818726, 0.5697224140167236, 0.3319162130355835, 1.1585575342178345, 1.6918809413909912, -0.24898996949195862, -0.824167788028717, 0.2735268771648407, -1.0727944374084473], [-0.20688992738723755, 0.5693630576133728, -0.17006804049015045, 0.29241931438446045, -0.49719905853271484, 0.7743358016014099, -1.1147067546844482, -0.5968642234802246, -1.700148582458496, -1.2828201055526733, -0.006842384580522776, -0.7035699486732483, 0.6981303095817566, -0.23596079647541046, 1.6971369981765747, -0.8306926488876343, -3.0121257305145264], [0.20908567309379578, 0.8622056245803833, 1.4288438558578491, -0.3564070761203766, -0.9191333055496216, 2.1920206546783447, -1.2072018384933472, 0.3285435140132904, 0.7622323632240295, -1.6885337829589844, 0.5407726168632507, 0.8003038763999939, 0.9032480716705322, -0.9334760904312134, 0.3132680058479309, -0.1715647578239441, -0.12818248569965363], [-0.10778699070215225, -0.5569037795066833, 0.895734965801239, 0.6963759064674377, 1.9872554540634155, -1.714127779006958, -1.2452577352523804, -1.0348379611968994, -1.643471360206604, 0.9406388401985168, -1.9590486288070679, -0.7563285827636719, 0.06372624635696411, -1.7315365076065063, -1.055915117263794, -1.7881048917770386, -0.90943443775177], [1.284592866897583, -1.1511493921279907, -0.6229742169380188, 0.0828169733285904, 0.0013320816215127707, -0.600027322769165, -0.3780655264854431, -0.08364962786436081, -0.8672687411308289, -0.0357869416475296, -1.4067182540893555, 0.22366462647914886, 0.8149780035018921, -1.143971562385559, -0.49793195724487305, 0.5405758619308472, -0.19101573526859283], [-0.5080130696296692, -1.9760667085647583, -0.9755042791366577, -0.06803574413061142, -1.133314609527588, -0.9144148826599121, -1.3799573183059692, 0.6274371147155762, -0.5705124139785767, -0.890070915222168, -0.41630980372428894, 2.054882287979126, 0.2064496874809265, -0.7427590489387512, -1.2447971105575562, 2.384965419769287, 0.1320633739233017], [0.3459552526473999, -0.8931774497032166, -0.27959129214286804, 0.5358865857124329, 0.46138930320739746, 0.5082647204399109, -0.6175963878631592, 0.9445480108261108, 0.3635961413383484, -0.05075601488351822, -2.419224977493286, 0.08905095607042313, 0.08871906250715256, -1.4114964008331299, 0.6622574329376221, 1.6497220993041992, -0.9933096766471863], [0.629753828048706, -1.5780632495880127, -0.9271959662437439, 2.732792377471924, 0.18735964596271515, 0.09270092844963074, -0.8237137794494629, -0.7794578075408936, -0.36218902468681335, 0.40270552039146423, 0.5290272831916809, 0.09366399794816971, -0.23845869302749634, -0.4882659614086151, -0.6074576377868652, -1.2424705028533936, -1.7632105350494385], [-0.46303799748420715, 0.38812628388404846, -0.8335763216018677, -0.27961111068725586, 0.8188333511352539, -0.008382135070860386, 0.851834774017334, -0.4358912706375122, -1.3242313861846924, -1.1186587810516357, 1.1059647798538208, 0.5718644857406616, 0.9533770084381104, 0.6597509384155273, 1.8791561126708984, -0.6361512541770935, -0.20287758111953735], [0.43889468908309937, 0.21187758445739746, 1.6245014667510986, -0.4100097119808197, -0.42807263135910034, -1.7087360620498657, 0.2070774883031845, -0.14540858566761017, -0.5004165172576904, -0.07367994636297226, -0.5529523491859436, 0.2223714143037796, 0.5467540621757507, 0.005021021701395512, 0.8471277952194214, -0.39228522777557373, -0.4788949191570282], [-0.12025655806064606, -1.580838918685913, 0.46579423546791077, -0.6152840256690979, 0.8896813988685608, 1.0559563636779785, -1.1819944381713867, -1.2125400304794312, 1.4441232681274414, -0.7460894584655762, -0.28106632828712463, -1.1014341115951538, -0.455596387386322, -0.34197503328323364, -1.1059819459915161, -0.1886778026819229, -2.1704018115997314], [-0.2543722987174988, -0.6977293491363525, -1.7210171222686768, -1.1937071084976196, -1.1299636363983154, -1.0733633041381836, -1.035569429397583, 0.42034298181533813, 0.28649094700813293, 1.3841657638549805, 0.9984172582626343, 0.10023064911365509, -1.2497860193252563, 0.399347722530365, -0.6715136766433716, -0.011300778016448021, 1.0714514255523682], [-0.29704615473747253, 2.3349835872650146, -0.062309470027685165, -0.7107598781585693, 0.04522582143545151, -0.8508273363113403, -1.1885923147201538, -1.1382513046264648, -0.5656993985176086, -0.509218156337738, -1.0734776258468628, 0.9582156538963318, -0.20807509124279022, -0.11005762219429016, -0.559188723564148, -0.20971335470676422, 0.49769192934036255], [-0.43799859285354614, -0.028113683685660362, 0.6340211033821106, 0.7833850383758545, 1.3583961725234985, -0.39975807070732117, 0.9398258924484253, -1.0510029792785645, 0.46407461166381836, 0.9695263504981995, 1.5705695152282715, 0.4745064079761505, 0.30963945388793945, -1.038528561592102, 0.557887613773346, -0.08380620926618576, 0.5037059783935547], [0.9856199622154236, -1.0432403087615967, 0.42753246426582336, -1.4557982683181763, -1.711418867111206, -0.4828681945800781, 0.10587921738624573, 0.01658874750137329, 0.25117096304893494, 0.20095126330852509, -1.6926878690719604, -1.607071042060852, 0.7497078776359558, 0.5981858372688293, -0.11365149915218353, -0.05946218967437744, -0.9309074282646179], [-0.8864702582359314, -0.15716435015201569, 1.415369987487793, -0.4318884313106537, 0.7395684719085693, -0.3281377851963043, 0.4489407539367676, 0.965916097164154, 0.6608320474624634, 0.15912450850009918, 0.6403327584266663, -0.3606468737125397, -0.07557137310504913, 0.19863636791706085, 0.5721362233161926, 0.6070508360862732, 0.5108744502067566], [-0.08696935325860977, 0.8067715167999268, 1.1458288431167603, 0.08681848645210266, 1.7351255416870117, -1.6487756967544556, -1.4412740468978882, 0.4144686758518219, -0.974271297454834, 0.5238893032073975, 0.7414478063583374, 0.44824281334877014, -1.5826693773269653, -1.3085963726043701, -0.7520366907119751, 0.37897050380706787, 0.0326978825032711], [-0.6677119731903076, 0.1396379917860031, -0.17033614218235016, 1.3057743310928345, -1.033921480178833, 0.44921809434890747, 1.270283818244934, 0.5229341983795166, -0.5399447083473206, -0.8080126047134399, -0.21795514225959778, 0.7153192162513733, 0.36679401993751526, 0.9360744953155518, -0.7792145013809204, 0.9586014747619629, -0.41125917434692383], [-0.23284879326820374, 0.1433277130126953, -2.5456056594848633, 1.8923084735870361, 0.4083654582500458, 2.0952229499816895, -0.8722571730613708, -1.267608404159546, -1.1228891611099243, 1.429967999458313, -0.6438958048820496, -0.9494014382362366, 0.6407372355461121, 0.15609341859817505, 1.0897897481918335, 1.46670663356781, 0.006278941407799721], [-0.9136385321617126, -0.9222944974899292, -1.6631958484649658, 0.20032364130020142, 0.7093170881271362, -0.6236118078231812, -1.5091511011123657, 0.34590449929237366, 0.16569846868515015, 2.3189148902893066, 0.5529212951660156, -0.9159077405929565, 0.870632529258728, -0.4541241526603699, -0.8732786774635315, -1.569928526878357, 0.20150041580200195], [-0.17749199271202087, -0.8469587564468384, 0.13233701884746552, -0.6088786125183105, 0.44316908717155457, -0.6847956776618958, -0.24906092882156372, -1.5495980978012085, -0.6559655666351318, 0.5328683853149414, -1.0636357069015503, -0.6048287153244019, 0.03627070412039757, 0.4082844853401184, -0.34141549468040466, 0.3327319622039795, -0.3297942876815796], [-1.4506717920303345, -0.42739686369895935, -0.17480072379112244, 0.8552286028862, 1.6943429708480835, 0.1157374158501625, 0.10978258401155472, -0.3937210142612457, 1.4224801063537598, -0.1311645209789276, -1.4534310102462769, 0.004676321521401405, -0.030001578852534294, 0.017901401966810226, -0.516327440738678, -1.1846743822097778, 0.3886493146419525], [1.8760831356048584, -1.7763856649398804, -0.3250780999660492, -0.8926810622215271, -0.7596347332000732, -0.34614992141723633, -0.6380762457847595, -0.8255894780158997, -0.15851831436157227, -1.2885550260543823, 1.7422198057174683, 0.7849573493003845, 0.32586219906806946, -0.6301966309547424, -0.7328148484230042, 0.9568412899971008, -0.4025229215621948], [0.2174290418624878, -2.0190646648406982, -1.6143170595169067, 0.21361301839351654, -0.7606601119041443, -1.5575920343399048, -0.5520427227020264, 0.9198698401451111, -0.1314690262079239, -0.8105325698852539, 1.9494445323944092, -1.8786488771438599, -0.8817269206047058, -0.9276356101036072, -0.8546810150146484, -1.1214202642440796, 1.4049650430679321], [0.3888779580593109, 1.8278611898422241, -1.169516682624817, -0.11602755635976791, -1.3939568996429443, 0.2209194153547287, -1.390959620475769, -0.5817413330078125, -0.6049680113792419, 1.1304759979248047, 0.04663599282503128, 1.082891821861267, -0.9702780246734619, 0.6574821472167969, -0.9200624227523804, -0.028448952361941338, -0.2747555077075958], [0.506714403629303, -0.019521834328770638, 2.2259604930877686, -1.061486840248108, 0.49598899483680725, -0.34688904881477356, -0.22466051578521729, -0.018659530207514763, 1.3262693881988525, 0.3873027265071869, -0.34280329942703247, -0.5847465395927429, -0.262753963470459, -0.9979307651519775, -2.2153031826019287, 1.0577033758163452, 0.5873703360557556], [0.2402324229478836, 1.6774661540985107, -0.49442043900489807, -1.3202258348464966, 0.271803617477417, -0.7452751994132996, 0.7538403868675232, -1.4488600492477417, -1.117340087890625, 0.04558084160089493, 0.34867873787879944, 0.6963708400726318, -0.05890694260597229, -0.2144802063703537, 0.896551251411438, -0.3436836004257202, 1.5091302394866943], [-0.04189401865005493, -0.30789312720298767, 0.6161221861839294, -0.02640519291162491, 1.1417648792266846, 1.034822940826416, 1.362234115600586, 2.093350410461426, 1.524397373199463, 1.2483007907867432, -0.9339599013328552, 0.5192436575889587, 1.0967949628829956, -0.8038605451583862, 1.0518187284469604, -1.9317573308944702, 0.15991060435771942], [-0.18852655589580536, -1.4777019023895264, 1.0478476285934448, -0.17881104350090027, 1.2741601467132568, -0.0394367091357708, -1.2241898775100708, 0.39678916335105896, -0.19706673920154572, -1.650252342224121, 0.301850825548172, 1.4976550340652466, 0.9304609894752502, -0.24060560762882233, -2.4861464500427246, 0.14326418936252594, 0.003958513494580984], [1.529641032218933, 0.20829477906227112, -0.6209628582000732, -0.3052877187728882, -1.1262940168380737, 0.05774686858057976, 0.928579568862915, 1.4552181959152222, -0.3271622359752655, -0.4903591573238373, 0.6425004601478577, 1.419634461402893, -1.9177320003509521, -1.388903260231018, -0.051842883229255676, 1.6911282539367676, -1.3529818058013916], [-1.303261399269104, 0.3607068657875061, 0.15272387862205505, 0.47260046005249023, 0.10596727579832077, -0.9058908820152283, -0.05829329043626785, -0.14348894357681274, -0.7928299903869629, -1.0727299451828003, 1.1601990461349487, -1.102614402770996, -0.33008694648742676, -0.9784296751022339, 1.217249870300293, 0.05960117280483246, -1.2180145978927612], [-1.2148973941802979, 1.1552001237869263, 0.044664256274700165, 0.028748303651809692, 1.3088817596435547, 1.3532692193984985, -1.0708422660827637, -0.5611130595207214, 1.0834906101226807, 0.3968062400817871, -0.4834146797657013, -0.4123111963272095, 0.06458181887865067, 1.8871515989303589, -0.20877660810947418, 0.3502986431121826, -3.3263821601867676], [0.5508641004562378, 0.2668452560901642, -0.048395831137895584, -1.2710764408111572, -0.634493350982666, 0.7566710710525513, 1.4268804788589478, 0.3097439110279083, 0.09615567326545715, 0.6494302749633789, 0.13614483177661896, -0.4930911064147949, -0.628121018409729, 2.2305376529693604, 0.24631841480731964, 1.8328888416290283, -0.5837777853012085], [-0.6187286376953125, 1.260554552078247, 1.4946038722991943, -0.08391844481229782, -1.303784728050232, 1.004185438156128, 1.7347127199172974, 0.14458787441253662, 2.5634682178497314, -0.3790207505226135, -0.21958665549755096, 1.050357699394226, 0.4770805835723877, 0.7578742504119873, -0.794013500213623, 1.1893290281295776, -0.501541018486023], [-0.7242206931114197, 0.6645628809928894, -0.9914751648902893, -0.1039167195558548, -0.14460840821266174, 0.15449638664722443, -0.12596525251865387, 1.028412938117981, -0.5798580646514893, -0.6865389347076416, -0.3682200610637665, -1.5076508522033691, -0.22190169990062714, -1.569161295890808, 0.0243962612003088, -0.10706747323274612, -0.700916051864624], [-1.0583930015563965, -0.5715247392654419, -0.6632595658302307, -1.1875756978988647, 1.3445308208465576, -0.2323731780052185, -0.0270904041826725, 0.17748478055000305, -0.2621133029460907, 0.0260782428085804, 0.11979581415653229, -0.102652408182621, 0.5573346614837646, -0.5001983046531677, 1.9352799654006958, -0.2793533205986023, -0.16736121475696564], [0.6618362665176392, -0.6543276906013489, -0.08909349888563156, -0.05769425258040428, 1.1361193656921387, 2.668052911758423, 2.7434141635894775, -1.2720988988876343, 2.333482265472412, 2.583122968673706, -1.2269433736801147, -0.8133978247642517, 1.8578892946243286, -1.7718710899353027, 0.9997152090072632, -0.3861438035964966, 0.8965702652931213], [0.5075821876525879, -1.1181834936141968, 0.07939759641885757, 1.3883622884750366, 0.16401556134223938, -0.10001400113105774, 0.07299244403839111, -2.4081122875213623, 2.928252935409546, 0.5443229079246521, -1.3379079103469849, -0.7775270938873291, 0.5743431448936462, -0.6585922241210938, -0.11969126760959625, -1.2990748882293701, 3.150682210922241], [0.9421744346618652, -0.10414260625839233, -0.6161516308784485, 0.4204515814781189, -0.6563277244567871, -0.9714435935020447, 0.6418398022651672, -1.144706130027771, 2.059473752975464, 0.019733916968107224, -1.5134351253509521, -0.6210494041442871, 0.5958067178726196, 0.40320509672164917, -0.7579925656318665, -0.12589262425899506, -0.6834878921508789], [0.4124979078769684, 1.439095139503479, 1.904477596282959, -0.4002760648727417, -1.2168400287628174, 1.550372838973999, 0.7598227262496948, 1.7205075025558472, -1.3944283723831177, 0.34747570753097534, -0.2287469357252121, -0.8809748888015747, 0.7999765276908875, 0.4668814539909363, 1.6400177478790283, -0.29600828886032104, -0.4907640516757965], [-0.6187127828598022, -1.765336036682129, -0.03907596319913864, -0.18038317561149597, 1.1456774473190308, 0.3926677405834198, -0.4094495177268982, 1.027308464050293, 1.034076452255249, 1.2982325553894043, 2.016047716140747, 0.3059443533420563, -1.2543003559112549, 0.3826570212841034, -0.4445776045322418, -0.043953102082014084, -0.8562294244766235], [0.6579616665840149, -1.9994032382965088, -2.3477783203125, -1.4547308683395386, 0.9407570958137512, -0.501728355884552, 0.5416038036346436, -1.6700652837753296, -0.8846673369407654, -0.4558604657649994, 0.38234591484069824, -1.6302356719970703, -0.5657661557197571, -1.637969970703125, 1.6649657487869263, -0.8143064379692078, 1.1212007999420166], [0.7173565626144409, -0.17828936874866486, -0.5915665626525879, 0.46842101216316223, 0.40514886379241943, -0.3395213484764099, 0.7839316129684448, -0.2743876278400421, -1.3782564401626587, 1.8341784477233887, 1.1888614892959595, 1.016898512840271, -1.6492102146148682, -0.2541487514972687, -1.0537502765655518, -0.29568013548851013, 0.3395368158817291], [-0.28736600279808044, 0.8870290517807007, 0.8815658688545227, -1.1270956993103027, 0.6089296936988831, 0.05518011748790741, -0.28140923380851746, -1.051368236541748, -1.413236379623413, -2.704511880874634, -0.5358450412750244, -1.2845675945281982, -0.45943883061408997, -0.8402172923088074, 0.07642052322626114, 1.314074158668518, -0.8163015246391296], [1.790827751159668, -1.7056595087051392, 0.23724545538425446, -0.9183785319328308, 1.133897066116333, 0.09817664325237274, 1.8970075845718384, -0.524053156375885, -0.8404008746147156, -0.697879433631897, -0.6183432936668396, 0.8755291700363159, 1.3485686779022217, -1.5047675371170044, 0.5899330973625183, -0.5292473435401917, -0.23652523756027222], [-1.4166767597198486, 2.5669496059417725, -1.0833656787872314, 1.2709362506866455, -0.3807268738746643, 0.12129479646682739, -0.8958901166915894, 0.42107829451560974, 1.0464718341827393, 0.3916409909725189, 0.015042620711028576, 1.6247811317443848, -0.7888119220733643, 0.20682986080646515, -0.5969116687774658, 0.9578376412391663, 0.3119122385978699], [-0.7088472247123718, 0.7496733069419861, 0.9050489068031311, 0.22803892195224762, 1.622484803199768, 0.891817033290863, 0.5153019428253174, -0.9351379871368408, -0.4870646297931671, 0.7473926544189453, -0.3622966408729553, -0.261155366897583, -0.3854111433029175, -1.0311174392700195, 0.25937318801879883, -0.6181389689445496, 0.40310347080230713], [-0.7916145324707031, 0.7602499723434448, -0.28520721197128296, -0.7833056449890137, 0.9569500684738159, -1.501269817352295, -0.5889368653297424, -0.181528702378273, -0.10619251430034637, -0.060393162071704865, 0.17636144161224365, -0.25478196144104004, -0.3627583384513855, -0.48397353291511536, 0.7162249088287354, -2.4907472133636475, -0.534424901008606], [0.4976441264152527, -0.2682713270187378, -1.396478533744812, 0.8349781632423401, -0.8412370085716248, -0.01105103176087141, 0.0554281584918499, 0.7751842141151428, -1.2141087055206299, 0.7574880719184875, 0.6317234039306641, -0.9629226922988892, 0.9842655658721924, -1.6371365785598755, -0.021200692281126976, -1.690902829170227, 0.9484585523605347], [-1.8701609373092651, 0.4498459994792938, -0.4395062029361725, -0.4946175813674927, -0.3039524257183075, -0.09271138161420822, -1.822005271911621, 1.3355803489685059, -1.2100932598114014, 1.2103629112243652, -0.8052282929420471, 0.3482459485530853, -0.21250365674495697, 1.2369343042373657, 0.9630846977233887, 0.0419650599360466, 0.8741397261619568], [-1.7029850482940674, -0.5695381164550781, -1.4532476663589478, 0.25270673632621765, -2.1667473316192627, -0.2013302445411682, -0.007591602858155966, -1.2761338949203491, -0.5632639527320862, -1.2496241331100464, 0.8385776877403259, -0.1135137528181076, 0.41760918498039246, 1.1391072273254395, -1.5231496095657349, -0.9497525691986084, 0.044750064611434937], [0.23466354608535767, 0.3410610258579254, 0.38638246059417725, -0.8095973134040833, 1.7193547487258911, -0.3042689263820648, 0.2159712165594101, 1.0906691551208496, -0.5271711945533752, -0.15664607286453247, 0.9560654759407043, -2.226585626602173, 0.8769012689590454, 1.0412250757217407, 0.2733803987503052, -0.8253869414329529, 0.6302806735038757], [1.5707132816314697, 0.17817747592926025, -0.23959937691688538, -0.6598436832427979, -0.5666830539703369, -0.3829924166202545, 0.28240081667900085, 0.11581338196992874, 1.4421513080596924, 0.8435400724411011, -1.2225910425186157, 0.46214592456817627, -1.1568790674209595, -0.06907562166452408, 0.9711031317710876, -1.3448892831802368, -2.4200217723846436], [0.0023832772858440876, 1.0502883195877075, 0.8768766522407532, 0.602357029914856, -0.029475772753357887, -0.5131695866584778, 0.7560974955558777, 1.3936619758605957, -0.6792315244674683, 0.3471129834651947, -1.345754623413086, 0.2060009241104126, -1.6495124101638794, -0.6760593056678772, 0.22805142402648926, -0.6000096201896667, 2.247020959854126], [1.0656442642211914, -0.38367390632629395, 0.6454532146453857, -2.6827759742736816, 0.4709489047527313, 0.6542690396308899, 1.8270782232284546, -0.28785938024520874, -0.1311892718076706, -0.46417516469955444, 0.7571825385093689, -0.026925919577479362, 0.17520053684711456, 0.9222218990325928, 0.461515873670578, -1.3019053936004639, -0.3265848457813263], [0.29110023379325867, -0.368890643119812, -0.007604074664413929, 1.558634877204895, -1.4808231592178345, -1.262064814567566, 0.13189548254013062, 0.21977598965168, 0.050930455327034, -0.08727188408374786, -1.7299773693084717, 0.5590187907218933, -0.41906559467315674, -0.9310570955276489, -0.020138908177614212, -0.5874019265174866, -0.580190122127533], [-0.20706267654895782, -0.7963047623634338, -0.8032577633857727, 0.8522424101829529, 1.1226099729537964, -0.6882151961326599, 1.1832565069198608, -0.7311228513717651, 1.3936952352523804, -0.3141038715839386, -0.12370817363262177, 2.294527053833008, 0.470897376537323, -0.8230351209640503, -0.23544958233833313, -0.41414397954940796, -0.41797277331352234], [0.8315396904945374, 0.18135392665863037, -0.9476604461669922, 2.807520627975464, -0.09370812028646469, 0.7447733879089355, -0.6045917272567749, 1.4950153827667236, 0.5352668762207031, 1.2372032403945923, -1.5835776329040527, 0.22295694053173065, 0.02146855741739273, 0.46154192090034485, -0.4334055185317993, -0.019147420302033424, 0.810757577419281], [1.5437871217727661, -1.4154473543167114, -0.12946376204490662, -1.8944357633590698, -0.5122315883636475, 0.24236372113227844, 0.13598030805587769, 0.8124558925628662, 0.39617598056793213, -0.43014809489250183, 0.39515748620033264, 0.7227749228477478, -0.923933207988739, 0.2211931198835373, 0.28917112946510315, -0.5936694145202637, -0.28669968247413635], [-1.1112672090530396, 0.46221110224723816, -0.592267632484436, -0.18474385142326355, -0.7788609862327576, 0.5651330351829529, 0.44031065702438354, 0.5509871244430542, -1.9381945133209229, 0.46605366468429565, 0.4758933484554291, -0.30326229333877563, 0.5687138438224792, 0.01157001405954361, 0.4961046576499939, 0.563748300075531, 0.9266850352287292], [-0.3123646676540375, -0.07967269420623779, 0.3027969002723694, -2.571943759918213, -0.8329306244850159, -0.7003037333488464, -0.7356975078582764, 1.2229841947555542, -0.19175399839878082, 3.591954469680786, 1.0069422721862793, 0.9963718056678772, 0.16486074030399323, -0.010991009883582592, 0.24513624608516693, -1.2878642082214355, 0.1769162118434906], [0.13569830358028412, -0.23038850724697113, -1.273007869720459, -1.5218898057937622, -1.2615091800689697, -1.267697811126709, 1.68504798412323, 0.9111600518226624, 0.337299644947052, -2.3399860858917236, -1.1476008892059326, -0.0025821353774517775, -0.8321569561958313, 0.9499085545539856, -0.5913722515106201, -0.8130708932876587, 0.18456006050109863], [-0.7857449650764465, 0.6154295802116394, -0.5298756957054138, -0.31393730640411377, -1.185546636581421, -0.7564272880554199, 2.2882468700408936, 0.006449138280004263, 1.815508484840393, 0.689967930316925, -0.18056423962116241, 1.3712577819824219, -0.014678806997835636, -0.27579066157341003, 0.5577648878097534, -0.8784375190734863, -0.45059165358543396], [1.459216833114624, -0.8227920532226562, 0.13196465373039246, 0.3965384364128113, -0.3558034300804138, 0.515283465385437, 0.01190719660371542, 0.730695366859436, -0.5518514513969421, 0.7221080660820007, -0.15814794600009918, -0.10581459850072861, 0.9053643345832825, -1.6481008529663086, 0.9273919463157654, 1.018402099609375, 1.677793264389038], [-1.7185471057891846, 1.9410734176635742, 0.59024977684021, 0.09409093111753464, -0.372021347284317, 1.7821645736694336, -0.39463570713996887, -0.03738552704453468, -1.3238050937652588, 1.2301630973815918, -0.6938359141349792, -0.20928920805454254, 0.593605101108551, -1.265140414237976, -1.2681055068969727, 1.235783576965332, 0.18840987980365753], [0.12207180261611938, -0.8333556056022644, -0.7594888806343079, -0.8191956877708435, -1.0676161050796509, -1.4264636039733887, 1.8778116703033447, -1.7017707824707031, 0.38824698328971863, 0.19145484268665314, 1.8474527597427368, 1.474250078201294, -0.5062277317047119, -0.45492586493492126, 0.028501583263278008, 0.8130769729614258, -0.641010582447052], [0.5615002512931824, 1.2103279829025269, -1.3743489980697632, -0.0054882485419511795, -0.12123364955186844, -0.37896448373794556, 0.9841752052307129, 0.4596453011035919, 1.9358248710632324, 0.855542004108429, 3.070896863937378, 0.42674216628074646, -0.7357873916625977, -0.1250194013118744, -0.2825881242752075, 1.149288535118103, -0.15525132417678833], [-0.03435836359858513, 1.5650122165679932, 0.24666321277618408, 0.05124586075544357, 2.6661393642425537, 0.9929205179214478, -0.1507217437028885, 1.2994155883789062, -1.9055376052856445, 0.8506823182106018, 0.5972933769226074, -0.1743900179862976, 1.4888172149658203, 1.611552357673645, 0.9675827026367188, -1.3725742101669312, 1.149662971496582], [0.4537448287010193, 1.0842530727386475, 0.1520179808139801, 0.05094541609287262, -1.5685596466064453, 0.9366469383239746, 1.8882153034210205, 0.6909135580062866, 0.22933702170848846, -0.3560042977333069, 1.1753004789352417, 0.8904269933700562, 0.04737654700875282, -0.9429369568824768, 1.9188511371612549, -1.070151448249817, -1.0971120595932007], [-0.40331941843032837, -1.091382384300232, 0.2591914236545563, 0.9000361561775208, 1.4619017839431763, -0.6286286115646362, 1.457784652709961, -1.2876628637313843, 1.0233980417251587, -1.2478495836257935, 0.6507729291915894, -1.3869364261627197, -0.21393653750419617, 0.7961004972457886, 1.3712214231491089, 1.1293443441390991, 0.10404451936483383], [-1.751006007194519, -0.2532571852207184, 0.9487301707267761, 0.33924680948257446, 0.08156793564558029, -0.693571150302887, -0.46899959444999695, -0.24626041948795319, 0.003823944367468357, 0.5222867727279663, -0.586563766002655, 1.3467167615890503, -1.2357558012008667, -0.8825596570968628, 0.09095936268568039, -0.3973183333873749, 0.7940584421157837], [0.33628562092781067, 0.8881076574325562, -0.7774419188499451, 0.6510133743286133, -0.27198436856269836, -1.4258009195327759, 0.8481999039649963, -0.744362473487854, 0.7543445229530334, -1.0436687469482422, 0.835915744304657, 1.0945452451705933, -1.9186360836029053, 1.5169882774353027, 1.4469279050827026, 0.25057730078697205, -1.6742292642593384], [-0.14639581739902496, -0.7208293080329895, 0.08842794597148895, 0.03490842506289482, -1.5832631587982178, -0.18341059982776642, 1.8121438026428223, 1.0517826080322266, -0.012407748028635979, -0.5440260171890259, -0.01865452527999878, 0.5725497603416443, -0.28626078367233276, -0.15730029344558716, -0.43391522765159607, -1.3670923709869385, 0.24971874058246613], [0.39574337005615234, 0.5265469551086426, 2.6860620975494385, -0.21639932692050934, 0.10819331556558609, 2.593953847885132, -0.3737098276615143, -1.1692357063293457, -0.7367346882820129, -1.326003074645996, -1.1443850994110107, 0.03753179311752319, -1.6902917623519897, 0.24340134859085083, 0.3156296908855438, -1.3168370723724365, 0.1956453174352646], [-0.8701199889183044, -0.9258202910423279, 1.5434452295303345, 1.0653059482574463, 0.19841594994068146, 0.49214300513267517, 2.2732110023498535, 0.32351747155189514, -2.089975595474243, -1.1204193830490112, 0.897902250289917, 0.08031119406223297, -1.2043243646621704, -1.717882752418518, 0.7310865521430969, -0.07633855193853378, -1.0312714576721191], [-0.3067404329776764, 0.33744174242019653, 1.532440185546875, 0.5375924110412598, 2.388078212738037, -0.07151178270578384, -0.8691415786743164, 0.9591647982597351, 1.5919560194015503, 0.9892036318778992, -0.8490874171257019, -0.8949764370918274, 1.25862717628479, 0.44112643599510193, -0.19684098660945892, -1.4462519884109497, -2.6362650394439697], [-0.3350885808467865, 2.5113883018493652, -0.7850003838539124, -1.1485369205474854, -0.32420626282691956, 0.002147075952962041, -1.1932878494262695, -1.0171658992767334, 0.839666485786438, 1.1294561624526978, -0.5264126062393188, 2.3970630168914795, -0.5312884449958801, 0.45665639638900757, 0.8770641684532166, -0.33642351627349854, -0.09476210922002792], [0.23076798021793365, -0.3301209509372711, 1.1088402271270752, -1.2859364748001099, -1.1798818111419678, 0.023766890168190002, 1.9194221496582031, -0.6076527833938599, -1.0233604907989502, -0.24284803867340088, 0.492436021566391, -2.2944226264953613, -0.4103820025920868, 0.328156977891922, 1.1200220584869385, -1.4715791940689087, -0.4358639717102051], [-0.20404791831970215, -1.3796695470809937, -0.24096737802028656, 1.7584903240203857, 0.23217976093292236, 1.1093125343322754, -1.1084064245224, 0.5225445628166199, 0.6996146440505981, 0.9126017689704895, 0.7593828439712524, -0.13202707469463348, 0.0977289080619812, 0.9089886546134949, -0.9731917977333069, 1.488482117652893, 0.3215703070163727], [-1.6778032779693604, -0.8861298561096191, 0.44170528650283813, -0.0671568363904953, -2.2016327381134033, -2.1406071186065674, 0.32064133882522583, 0.6278818845748901, -1.1363805532455444, -0.47540146112442017, -1.4991271495819092, -0.0354282483458519, -0.03683747351169586, -1.315480351448059, -1.6554291248321533, 0.5308752655982971, -0.08837059885263443], [-0.36289873719215393, -0.1161300539970398, -0.8860286474227905, -1.091694951057434, -0.9138927459716797, -0.4020310044288635, -1.9652045965194702, -1.428011178970337, -0.36438795924186707, -0.6441442370414734, -2.4388468265533447, -0.44815513491630554, 0.5466815829277039, -1.3878599405288696, -0.5065005421638489, -1.4414912462234497, 2.0375614166259766], [-1.0565812587738037, 0.4214712977409363, -0.631678581237793, -1.7335116863250732, 1.3511954545974731, -1.2492105960845947, -0.8917326331138611, 0.16050049662590027, -1.6630898714065552, 1.1168038845062256, 0.40855351090431213, 0.39207613468170166, 1.305077314376831, -0.14047564566135406, -0.34083592891693115, -0.24161577224731445, 0.4616607129573822], [0.047984424978494644, 1.1942360401153564, -0.3543899655342102, -1.1010844707489014, 0.5418508648872375, -0.26012101769447327, 0.6701743006706238, -1.7727845907211304, 0.1004633903503418, 0.22455339133739471, -1.1836509704589844, -0.25526782870292664, -0.22980061173439026, 0.9589068293571472, -0.29614731669425964, -1.969939112663269, 0.07868143916130066], [-0.10929813981056213, -1.178661584854126, 1.4192384481430054, 0.9811092615127563, 0.26769518852233887, -0.4511844217777252, -0.6081098914146423, 2.0083069801330566, 0.0008596867555752397, -0.6648229956626892, -1.111732840538025, -0.6709129214286804, 0.12735866010189056, 0.5323346257209778, -1.2275185585021973, 0.28405141830444336, 0.24557073414325714], [2.178187847137451, -1.1995699405670166, -0.2841967046260834, 0.6822857856750488, 0.9632237553596497, 1.729854702949524, -2.0696709156036377, 1.5455435514450073, 0.219354510307312, -0.5234962105751038, 1.2451863288879395, 0.9933491945266724, -0.6456536054611206, 1.001743197441101, -1.8702231645584106, -0.3577329218387604, -0.9685409665107727], [0.17015154659748077, -0.8139857053756714, -0.1751348376274109, -0.8866559863090515, 1.3951284885406494, -1.449445366859436, -0.29533106088638306, 1.8286845684051514, -1.3656713962554932, 0.4900473952293396, -0.754105269908905, -0.5495683550834656, -1.5048837661743164, 2.3461854457855225, -0.59473717212677, -0.8487223982810974, -0.49937504529953003], [-1.4816142320632935, 1.1791865825653076, -1.1834057569503784, -0.2935042977333069, 0.3771950602531433, 0.6867528557777405, -1.1455062627792358, -0.6520900130271912, -0.2470281720161438, -1.564537763595581, 0.9670577645301819, -0.23210857808589935, 0.1729022115468979, -0.10451601445674896, 0.5210029482841492, -1.28505539894104, 1.274637222290039], [1.6513583660125732, 0.3366392254829407, -0.33088743686676025, -0.5686336159706116, 0.3297901153564453, -0.44479113817214966, 0.976124107837677, -2.554586410522461, 0.5530560612678528, -2.2481961250305176, 0.41673484444618225, 0.6991471648216248, -1.2626018524169922, -0.19374750554561615, -1.5619324445724487, -0.5436941385269165, 0.861696183681488], [-0.06471478939056396, -0.8881930112838745, -0.9013589024543762, -1.0081701278686523, -0.36058467626571655, 2.424766778945923, -0.9032522439956665, 0.520697295665741, -1.9568523168563843, 2.438412666320801, -0.0979548916220665, -0.7496737241744995, 0.6968802809715271, -1.4955406188964844, -0.9996532201766968, 0.06193206459283829, 0.5144995450973511], [-0.16313442587852478, 1.1233102083206177, -0.5768600106239319, 1.0273948907852173, 0.0854845643043518, -0.8308971524238586, -0.1628805696964264, 0.5815848112106323, -0.2837260365486145, 1.7256417274475098, 2.156311511993408, -0.4130772650241852, 0.8998317718505859, 1.698783278465271, -2.902153730392456, -0.47532182931900024, 1.1313365697860718], [0.750525712966919, -0.487430602312088, 0.7957653403282166, 0.15392984449863434, -1.5724233388900757, 0.824396014213562, -0.29426634311676025, -0.716352105140686, 0.2639092803001404, 1.301957130432129, 0.3436967134475708, -0.3707970082759857, 0.5396543741226196, 0.8684252500534058, -1.4522697925567627, -2.280579090118408, -0.5373665690422058], [-0.7288921475410461, -1.1280564069747925, -0.29691317677497864, 1.7153886556625366, -1.108203411102295, 0.5363553762435913, -1.9944584369659424, -0.8489058017730713, -0.7490027546882629, 1.2966701984405518, -0.38290151953697205, -1.038102149963379, -0.6540466547012329, -0.02161518484354019, 0.4589983820915222, 0.6881272792816162, 0.9957299828529358], [0.5904966592788696, 1.0481661558151245, 0.43001124262809753, 0.21807223558425903, 0.3315894603729248, -0.40924137830734253, 0.7751803994178772, -1.1875289678573608, 0.2826363146305084, 1.0560485124588013, -2.296206474304199, 0.4622935652732849, 0.43935590982437134, 1.10218346118927, 1.3606783151626587, 1.4481068849563599, -1.508827805519104], [0.0111205093562603, 0.20721645653247833, 0.8260641098022461, -1.3470790386199951, 0.4717489182949066, -0.7824072241783142, 0.5223166346549988, -0.39385542273521423, 0.338533490896225, -0.2639632523059845, 0.8488019704818726, 0.9581927061080933, 1.3434040546417236, -0.08338242024183273, -0.4204583168029785, -0.7571637034416199, 1.963025450706482], [-0.21858538687229156, 0.34812310338020325, 0.4854663908481598, -1.4625253677368164, 0.8242970108985901, -1.6905722618103027, -0.15411920845508575, -1.0651068687438965, 0.22947612404823303, -0.7924011945724487, 0.8764359951019287, -1.6381253004074097, 0.7927044630050659, -1.256797432899475, -1.1045295000076294, -1.1557515859603882, -0.3460073173046112], [0.03205965831875801, 0.598743736743927, -1.759043574333191, 0.8048679232597351, -1.0475280284881592, 1.1733158826828003, -0.9229965209960938, 0.4234243631362915, 0.26580551266670227, -0.5368244647979736, -0.27790772914886475, -0.9500687718391418, -1.2701547145843506, 0.27660760283470154, -0.21058279275894165, -0.987296462059021, -1.7011876106262207], [0.30095982551574707, -0.21444065868854523, 0.5666928291320801, 0.4645799398422241, -0.37738680839538574, -0.9735350012779236, -0.4638385772705078, -0.3755002021789551, -0.019232837483286858, -1.4028398990631104, -0.6723505854606628, -1.387461543083191, -1.755508542060852, 0.7926673889160156, 0.2530411183834076, -0.4748891592025757, -1.5750572681427002], [-0.7158185243606567, 0.43190619349479675, -0.685270369052887, 0.7163972854614258, 1.906448245048523, -0.04932565614581108, 0.6172183752059937, -1.1538559198379517, -1.2398663759231567, -1.6944172382354736, 0.8486843109130859, 0.8483065366744995, 0.36335116624832153, -1.5765894651412964, 1.1973748207092285, -0.3951409161090851, -1.1464205980300903], [-0.08604356646537781, -0.7131572365760803, 1.5575370788574219, 0.831128716468811, 0.28832918405532837, 0.07625166326761246, 1.0125552415847778, 1.0758174657821655, 1.690738558769226, 0.06600866466760635, -0.8388645052909851, 0.03968549892306328, -0.7944585084915161, 1.6625816822052002, -0.23576302826404572, 0.6369481682777405, -0.5871497392654419], [0.03605449199676514, 0.18169420957565308, -0.54544597864151, -1.2148877382278442, -1.1888635158538818, 0.5610007047653198, -0.04264216870069504, 0.5350399017333984, -1.5498261451721191, -1.19330632686615, -0.9414661526679993, -2.100522041320801, 0.2815941274166107, 0.8092136383056641, -0.7550731301307678, 1.2400290966033936, 0.47700148820877075], [0.4925062358379364, -0.08207189291715622, -0.4992872178554535, 0.37468814849853516, 0.08480191975831985, -2.0872795581817627, -0.7605977654457092, -1.1159225702285767, -0.021876156330108643, 0.5746931433677673, -0.2338259369134903, 0.13310928642749786, -1.0627890825271606, 0.33144497871398926, 1.3969751596450806, 1.1185991764068604, -0.31005486845970154], [0.844415009021759, 2.4210171699523926, -0.4789890944957733, 1.108490228652954, 0.7897017598152161, 0.3470882773399353, -0.441934734582901, -0.7428076863288879, 0.23919662833213806, 0.32043859362602234, -0.5702870488166809, -0.2203494757413864, 0.2901289463043213, 0.7722539901733398, 0.8074215054512024, 1.235450267791748, -0.12035006284713745], [-0.3278539180755615, -0.7663519978523254, 0.2025003880262375, 0.005583535879850388, -1.589437484741211, -0.10827510058879852, -0.03360053151845932, 0.3579910397529602, -0.8851248025894165, -1.6679227352142334, 0.30200475454330444, 0.6326629519462585, -0.9229806661605835, -0.3709211051464081, 0.1629982739686966, 1.1627789735794067, 0.6727558970451355], [0.04003281518816948, 0.257373571395874, 0.5503085851669312, 0.4123448431491852, -0.3333927094936371, 0.2115577757358551, -0.20060640573501587, -1.0249935388565063, -0.14912649989128113, 0.9806464910507202, 0.7881369590759277, -0.6834577918052673, -0.30868270993232727, -0.13819314539432526, -0.3262498676776886, 0.92237389087677, -0.7775349617004395], [0.666281521320343, 1.684415578842163, 0.5007855296134949, -1.0660593509674072, -0.890667200088501, -1.2343703508377075, 2.0755438804626465, 0.7183929085731506, -0.4718160033226013, 1.5123677253723145, -1.441202998161316, 1.9960026741027832, 0.495572566986084, -0.31252288818359375, 0.14050140976905823, 1.3215173482894897, 1.7925810813903809], [-2.0309038162231445, 0.5194193124771118, -0.889630913734436, 0.31983616948127747, 0.6135636568069458, 0.3906419277191162, 1.020119547843933, -1.368679404258728, -2.0893359184265137, 0.44393470883369446, -1.5620397329330444, -0.2803964912891388, -0.879455029964447, 0.35127657651901245, 0.03523556515574455, -0.9203563928604126, -1.3443918228149414], [1.420412540435791, -0.23184724152088165, 2.2470738887786865, -0.008766301907598972, 1.1373976469039917, -1.0520292520523071, -1.2306386232376099, -0.5505589246749878, -0.024587837979197502, 0.7111576795578003, -0.28446677327156067, -0.01613449677824974, 0.6555168628692627, 1.1072406768798828, 1.9449416399002075, 0.5324528217315674, -0.0692482590675354], [-0.9399131536483765, 0.6563925743103027, -0.3925490379333496, 0.4331165850162506, 1.118972897529602, -0.622439444065094, 0.05467937886714935, -0.10674352198839188, 1.0019224882125854, -0.5538328289985657, -2.1380321979522705, -1.7529900074005127, 1.1032719612121582, 0.2559588551521301, 1.56777024269104, -0.07832080870866776, -3.2048110961914062], [-1.9448033571243286, -0.5164036750793457, -0.6670206189155579, -1.7133244276046753, -0.31746503710746765, -0.629080057144165, 0.033154964447021484, -1.8220034837722778, -1.9401627779006958, 0.6168050765991211, -1.2040283679962158, 1.1751757860183716, 1.3011373281478882, -0.37799960374832153, 0.6735632419586182, -0.7594313025474548, -0.1685524433851242], [0.6476092338562012, -0.9486557841300964, 0.9321387410163879, 1.6873064041137695, -0.21104730665683746, 1.2635663747787476, 0.8395587801933289, 0.7658712267875671, -0.6063597798347473, -0.08456721156835556, 0.6788158416748047, -1.5437824726104736, 1.3334100246429443, -1.608717918395996, -0.2579374611377716, -0.5198675990104675, -2.612107992172241], [0.17287853360176086, -0.018580907955765724, -1.6687930822372437, -0.5624046921730042, -1.613215446472168, 1.7424578666687012, -0.4313226342201233, -1.137855052947998, -1.5251855850219727, 0.5308043360710144, -0.7032204270362854, 0.1059938296675682, 0.6208671927452087, 0.47758299112319946, 0.9504686594009399, -0.5064879059791565, 0.4431890845298767], [2.5437934398651123, -0.07948990166187286, 0.1897965967655182, -0.7742895483970642, 1.156455397605896, 1.1152122020721436, -0.961493730545044, 0.14174439013004303, -0.2609376609325409, -0.05309139937162399, 1.1871662139892578, -0.4033912122249603, -0.7684106826782227, -0.7300844788551331, -0.3752807378768921, 1.2029836177825928, 0.18325723707675934], [1.1194978952407837, 1.3229548931121826, 0.669203519821167, -1.6719602346420288, 1.5502911806106567, 0.862822949886322, -0.7447903752326965, 0.8162699937820435, -0.15352028608322144, -0.10061608999967575, -0.18833576142787933, -0.1885572224855423, 0.5465236902236938, 0.40435177087783813, 0.3317846655845642, 0.19291554391384125, -0.7187454700469971], [-0.9374710917472839, 0.35306715965270996, 1.9991035461425781, 1.588938593864441, -0.49985480308532715, -0.5866059064865112, 0.5226762890815735, -0.1655665636062622, -1.0436069965362549, -0.31365764141082764, 0.6516549587249756, -1.862855076789856, -0.6527795791625977, -0.43675509095191956, -2.2130157947540283, 2.1776044368743896, 0.9801746606826782], [-0.5486624836921692, 3.8491311073303223, 1.24195396900177, -0.6931478381156921, -1.2034605741500854, 2.0635275840759277, -0.6432901620864868, 0.11246640980243683, 0.7549879550933838, 1.1319036483764648, 1.3910270929336548, -0.8949228525161743, 0.38160234689712524, -0.465074747800827, -2.2918989658355713, 1.7035541534423828, 0.17288999259471893], [2.697319984436035, -0.5111251473426819, 0.7043778896331787, -0.3874873220920563, -0.3821899890899658, 0.4369361698627472, -1.0136826038360596, 1.1951245069503784, 0.1754656583070755, -0.7774341106414795, 1.4648473262786865, -0.9402818083763123, -0.5791444778442383, 0.8045569062232971, -0.05256145820021629, -0.5893437266349792, 0.7047310471534729], [-0.9368098974227905, 0.24894754588603973, -0.1360156536102295, 0.3600827157497406, 0.4170285761356354, 0.7539191246032715, -0.060939956456422806, 0.036016423255205154, -1.39141047000885, 2.3262040615081787, -0.19166286289691925, -0.8197876811027527, 0.6162667274475098, 0.8941149711608887, 0.09075918048620224, 0.3496280610561371, 1.3116170167922974], [-0.388868123292923, 0.3339482843875885, -0.2607704699039459, -0.14800496399402618, -0.034497667104005814, 0.25463613867759705, 0.06912004947662354, 0.6894001364707947, 0.1308489590883255, 0.03915412724018097, -0.8976300954818726, 1.5493109226226807, -0.47341427206993103, -1.1859543323516846, -0.4914168119430542, -1.150510549545288, -0.11467810720205307], [-0.4759311378002167, 0.07840575277805328, 1.0234365463256836, -1.4280861616134644, -3.433199644088745, 0.30870071053504944, -1.1791753768920898, 0.30726954340934753, -2.0485527515411377, -0.25139737129211426, 1.4043596982955933, 1.614396333694458, 0.5206073522567749, 0.35836300253868103, 0.43697988986968994, 0.015101101249456406, -0.35833609104156494], [-1.11210298538208, 0.017022576183080673, 1.8780113458633423, -0.721335768699646, -1.3386602401733398, -0.3621918559074402, 0.8242264986038208, -1.6652685403823853, -1.7684388160705566, 0.12555454671382904, -0.34389546513557434, 1.9545000791549683, 0.3265887498855591, -0.05479762703180313, -0.7250534892082214, 1.2370102405548096, 1.6013214588165283], [-0.13621622323989868, -0.09569214284420013, 0.49246346950531006, 1.2965624332427979, 0.43972834944725037, -1.0532472133636475, 0.540947675704956, -1.6997647285461426, -0.3650592267513275, 2.4478328227996826, 1.967793583869934, 0.594689667224884, 0.45102837681770325, 1.5313482284545898, 0.9928275346755981, -0.29582399129867554, -0.30713269114494324], [-1.674129605293274, -1.0638868808746338, 0.141596719622612, -0.9208346009254456, -0.3505955636501312, 0.330824077129364, -0.3313332498073578, 1.58601713180542, -1.059822678565979, 0.8204541802406311, 0.9237933158874512, 0.0065199630334973335, 1.244657278060913, 0.3387553095817566, 0.18701492249965668, -0.048192206770181656, 1.8408795595169067], [0.20830106735229492, 0.38768771290779114, -1.8977854251861572, -1.6668452024459839, 0.9130883812904358, -0.23318229615688324, -0.6961095929145813, -0.5553757548332214, 2.387805461883545, -0.11502660065889359, -1.4951698780059814, 2.354377269744873, -0.6892812252044678, 0.9967695474624634, 1.128143548965454, -0.5101197361946106, -1.272844672203064], [0.8374410271644592, 1.974020004272461, -1.3298146724700928, 0.6438676714897156, -0.19125141203403473, -1.4387211799621582, -0.4469752609729767, -0.8433937430381775, 0.45767152309417725, 0.3455757200717926, 0.05900418385863304, -0.23940253257751465, -1.6637877225875854, 0.24673591554164886, -0.47071895003318787, 0.3301197290420532, 0.586695671081543], [-0.2106379270553589, -1.0309524536132812, -1.7269363403320312, 1.0391359329223633, -0.6092408895492554, 0.2056644856929779, -0.43625152111053467, 0.684619665145874, 1.3789384365081787, 0.23174075782299042, 0.35472571849823, -1.0992811918258667, -0.35965636372566223, -1.6549185514450073, 0.2551995515823364, -0.9502599239349365, 1.1528739929199219], [0.48706522583961487, 1.7841179370880127, -1.1436936855316162, 0.2134217619895935, 0.37021511793136597, 0.5049266815185547, -0.8102000951766968, -0.15852895379066467, 0.7105932831764221, 0.8796839714050293, 1.107561707496643, 2.189189910888672, -0.6099540591239929, 0.0485711432993412, 1.0217006206512451, -0.5172808170318604, 1.0309971570968628], [3.057086944580078, 0.6777901649475098, -0.18822601437568665, 2.083883285522461, -0.15885955095291138, -0.0008076181402429938, 0.24465271830558777, 1.4403125047683716, -0.7108995318412781, -1.033392071723938, 0.4120441973209381, -0.5190877914428711, 1.3193936347961426, 0.2620293200016022, 0.21788087487220764, 0.19793075323104858, 0.4137193262577057], [-0.4272092580795288, -1.094242811203003, 0.47799059748649597, -1.4934426546096802, -0.533511757850647, 1.8143707513809204, 1.3863861560821533, -0.7156252861022949, 0.8853045105934143, 0.1021069660782814, 0.8911595344543457, 1.1605539321899414, -1.3696105480194092, 0.2691790461540222, 0.7809121012687683, -0.47434377670288086, 0.22534844279289246], [-1.5432072877883911, 0.7320968508720398, 0.7570295929908752, 0.5120844841003418, -0.7920682430267334, -0.33783185482025146, 0.44576558470726013, -1.0733243227005005, -1.2461563348770142, 0.1586250215768814, 0.6291158199310303, 0.1628231257200241, -0.11208042502403259, 0.23452244699001312, 1.0656404495239258, -0.5045408606529236, 0.6816656589508057], [-0.35115954279899597, 0.805134117603302, 0.2737298011779785, -1.0467231273651123, -1.2984321117401123, 0.7072094678878784, 0.6210476160049438, -0.7885653376579285, -1.1387174129486084, 2.1290290355682373, -0.34767085313796997, -1.3778992891311646, 0.2317924201488495, 0.14673225581645966, 0.5132005214691162, -0.1901542991399765, -1.0797019004821777], [1.1392741203308105, 1.122204065322876, 0.8194270730018616, 1.0367504358291626, -0.4221069812774658, -0.8650664687156677, 1.1030017137527466, -0.24364790320396423, 0.8609526753425598, 1.7742785215377808, 0.823954701423645, 0.1221342533826828, -0.4971167743206024, -1.9270256757736206, -0.14654529094696045, -2.2712059020996094, -1.229973316192627], [-0.37307408452033997, 0.003560758428648114, -0.7623341083526611, 1.7133665084838867, 0.9296715259552002, 0.5809897184371948, 0.589853048324585, -0.4749860465526581, 1.1304779052734375, 0.8854948282241821, -1.5752578973770142, -0.5690069794654846, 0.700579822063446, 0.8569914698600769, 0.1261330246925354, -0.3639158308506012, 0.06270504742860794], [-0.5451949834823608, -0.7285445928573608, -0.08089388906955719, 0.8694795370101929, -0.6087178587913513, 0.7090206146240234, -2.056898355484009, -0.15623727440834045, -0.660014271736145, 0.444030225276947, 0.2900301516056061, -0.8401439189910889, 0.8762356042861938, -0.6447171568870544, 0.7198407649993896, -1.5092655420303345, -0.8002040386199951], [0.16800914704799652, -0.31731942296028137, 0.884405791759491, -0.4179021716117859, 1.3461626768112183, -0.5974194407463074, 0.21887817978858948, -0.7209808230400085, -0.07488200068473816, -1.5931460857391357, 0.9210964441299438, 0.5474856495857239, -1.4668079614639282, 1.8812097311019897, 0.4989781677722931, -0.8985406756401062, -1.080353021621704], [0.3778074383735657, -0.7324962019920349, 0.029624266549944878, -0.566601574420929, -0.9628263711929321, 1.078242540359497, 1.7106621265411377, -0.24472391605377197, 0.8510469198226929, 0.7648559808731079, -0.18503841757774353, 0.3484800457954407, -0.6305578351020813, 0.3485591411590576, -0.024646446108818054, 0.33627668023109436, -2.116271734237671], [0.44482937455177307, -0.47296929359436035, -2.6538004875183105, 0.729897141456604, 1.8947640657424927, 0.4828581213951111, 0.670659601688385, 3.4846136569976807, -2.4924700260162354, -1.058587908744812, -1.6534063816070557, 0.9407116174697876, 0.36619099974632263, -0.06331349164247513, -3.468806743621826, -1.4519304037094116, 0.6932478547096252], [-2.055793046951294, -1.2927119731903076, -0.15364579856395721, -0.4472908675670624, -1.2636531591415405, 2.0967304706573486, -0.09616834670305252, -0.02129518799483776, -0.05310801416635513, -1.5385966300964355, 2.288722276687622, 0.9526658654212952, -0.7926740646362305, -0.8421717882156372, -0.35000157356262207, -1.5833834409713745, 0.7475539445877075], [-0.009598827920854092, 0.2308720499277115, -0.6351132988929749, 0.8936729431152344, -0.3245047628879547, -0.6430355906486511, -0.5264288187026978, -0.2659187912940979, 0.9559917449951172, -0.47841134667396545, -1.1635386943817139, 0.9868776202201843, 0.44876158237457275, -2.076873779296875, 0.4197184443473816, 0.6518659591674805, 0.7051352858543396], [-0.3304084539413452, -0.03322245180606842, -0.3924659192562103, 0.337249219417572, 0.29278865456581116, -2.0623550415039062, -0.10103186964988708, 0.49962738156318665, 0.2786964774131775, -0.33101415634155273, -1.5648999214172363, -1.6659878492355347, -0.48950713872909546, 1.3618953227996826, 0.7213740944862366, 0.5289973020553589, 0.9652575254440308], [0.37905964255332947, -0.06922448426485062, 0.2190239429473877, -2.8571717739105225, 0.7494006752967834, 0.2548082768917084, 0.545731782913208, 1.3092325925827026, -1.9608083963394165, 1.3029435873031616, -0.4596509039402008, -0.7677927017211914, 1.9030563831329346, 0.26289352774620056, -2.521930694580078, 0.585306704044342, 1.0766879320144653], [-0.039079081267118454, 2.485341787338257, -0.2838304340839386, 0.9540795087814331, -0.06732794642448425, 1.0824764966964722, 1.0724170207977295, 0.8857089281082153, 1.195273756980896, 0.5529122948646545, 1.6677143573760986, 0.7796902656555176, -0.7877637147903442, -0.9734289646148682, 1.1420142650604248, -0.6343274116516113, -0.9893960356712341], [0.2851986587047577, -0.17357228696346283, -1.1322134733200073, 1.1654311418533325, -1.2663078308105469, 0.3638419806957245, 0.8663827180862427, 0.6711881756782532, 0.47712722420692444, -0.7400917410850525, 0.29459449648857117, -0.6168466210365295, 0.2756468951702118, 0.7372552156448364, -0.8644960522651672, 2.3957295417785645, 0.027457213029265404], [2.1425554752349854, 1.05278480052948, 0.5299957990646362, 0.886708676815033, 0.08781479299068451, -1.1448746919631958, -0.20427124202251434, -0.8137244582176208, 0.2024979293346405, 0.5101284384727478, 0.7796337604522705, -0.7298376560211182, -0.97153639793396, -0.23025639355182648, 0.554969072341919, 0.40729770064353943, -1.5654164552688599], [1.851361870765686, -2.004162073135376, -0.366482138633728, 1.2909437417984009, 2.0887033939361572, 0.24739788472652435, -0.30439406633377075, 0.4214714467525482, -1.1537373065948486, 0.2764054238796234, -1.182662010192871, -0.6686594486236572, -0.7997040152549744, -0.9768041968345642, 1.087246298789978, 1.5645257234573364, 0.664053201675415], [-0.22543753683567047, 0.4076356887817383, 2.2325894832611084, -0.4991694390773773, -0.2494058758020401, 0.057534683495759964, -0.2822794020175934, -1.17867910861969, -0.7894244194030762, 1.490893840789795, 0.7772496938705444, -1.894952654838562, 0.3169141709804535, -0.0050771101377904415, 1.7368611097335815, 1.3863245248794556, 0.16215333342552185], [0.7038982510566711, 2.4451146125793457, 1.1872912645339966, 1.5395054817199707, -1.1645642518997192, 1.223605990409851, 0.8328794240951538, 0.018964329734444618, -0.5601911544799805, -0.23356153070926666, 1.0363579988479614, 1.1475365161895752, 0.18554431200027466, 0.5267785787582397, -0.6663092374801636, -0.9001796245574951, -0.9231443405151367]], "dtype": "float32"}, {"__nd__": [-0.7130190134048462, -0.018563242629170418, -0.6572844386100769, 0.1019752025604248, 0.5376960039138794, 0.47242555022239685, -0.7129987478256226, -0.77821284532547, -1.7475279569625854, 0.8462565541267395, -0.8111661672592163, -0.911961019039154, -0.5608220100402832, -1.651494026184082, 1.0756916999816895, 1.2174277305603027, 0.49129366874694824], "dtype": "float32"}, {"__nd__": [-0.041468940675258636, 1.0276950597763062, -0.643409788608551, -0.2793235182762146, -0.4526462256908417, -1.4053699970245361, -0.0022294449154287577, 0.149578258395195, 0.20825037360191345, 0.5066705346107483, -0.3776727318763733, 0.3505339026451111, 0.23858098685741425, -0.6744391322135925, 0.9736637473106384, -0.3322656750679016, -0.9171004295349121, -0.08922275900840759, -1.4328603744506836, -2.519333839416504, -1.3159561157226562, -1.1782430410385132, -0.9964653253555298, 0.3456072211265564, 0.8500604629516602, 0.07953329384326935, -0.5780819058418274, 0.12176791578531265, -2.076333999633789, -0.25305885076522827, -0.2953365445137024, -0.9648661017417908, -1.1042729616165161, 0.3259289264678955, -0.29509520530700684, 0.5513931512832642, -0.5076076984405518, -0.58440101146698, 0.06400037556886673, 2.736872911453247, 0.353939950466156, 1.0407015085220337, 0.1968299299478531, 2.2662413120269775, 0.5629693865776062, 0.3629745543003082, -0.7270780801773071, -0.8298056125640869, -1.3653656244277954, -0.12199270725250244, 0.9424917697906494, 0.35672125220298767, -0.9367097020149231, -1.3603309392929077, -1.256941318511963, -0.10513973981142044, 0.01576077565550804, 0.4951958656311035, -1.1766881942749023, -0.07649771869182587, -1.1615198850631714, -0.6386467814445496, -0.8556683659553528, -1.513992428779602, -0.018451008945703506, -0.2403431534767151, 0.9345976114273071, -0.14840702712535858, 0.43573322892189026, -1.2862603664398193, -0.8448907732963562, -0.5081442594528198, -2.1562507152557373, -1.4437755346298218, -0.6624037623405457, 1.284533977508545, 0.17675846815109253, -0.03804286569356918, 0.32231438159942627, 0.7981410622596741, 2.1738154888153076, 0.2703574299812317, -1.284183144569397, -0.4926624298095703, -0.3742089867591858, 0.8047046065330505, -0.10911709070205688, 0.20997188985347748, -0.45990189909935, -0.2061343640089035, 0.4389965236186981, -1.124298334121704, 0.556390643119812, 0.036144331097602844, -1.5538239479064941, 0.5546126961708069, -0.8668515086174011, -0.38499701023101807, 0.4810670018196106, -1.0105940103530884, -1.033998966217041, -1.0411639213562012, 0.2330564707517624, -0.15885235369205475, 0.11127985268831253, -0.6273279190063477, -1.6063939332962036, 0.6196956038475037, -0.6221632957458496, -3.6097607612609863, 0.5640417337417603, -0.8486453890800476, -1.00881826877594, 1.544589877128601, -0.5111271739006042, 1.641658902168274, 1.1671667098999023, 1.7250746488571167, 1.429715633392334, 1.1895043849945068, -1.1462618112564087, -0.2743243873119354, -1.0912230014801025, 0.808961033821106, -0.960640013217926, -2.5085415840148926, -0.9215098023414612, 0.17389258742332458, -2.356590509414673, 0.17347978055477142, -1.2710299491882324, 0.8238223791122437, -1.1683170795440674, 1.422716498374939, 0.15414643287658691, -0.2500225901603699, 0.39842841029167175, -0.686301052570343, -0.21301355957984924, -1.1978439092636108, 0.17098551988601685, 0.8497352600097656, 0.6119204759597778, -1.833748698234558, 0.557176411151886, -0.2629750072956085, -0.9288095831871033, -0.18890997767448425, 1.7607650756835938, 0.1509607434272766, -0.28081029653549194, 0.33002516627311707, -1.0839370489120483, -1.0773580074310303, -0.9921172261238098, 0.5757057666778564, -0.8393327593803406, -0.06255511939525604, -0.3248685896396637, -2.0444905757904053, 0.9423717856407166, 1.1178451776504517, 0.19777902960777283, -1.2441492080688477, 0.08049530535936356, -0.08879835158586502, -0.6132573485374451, -0.39646270871162415, 0.549696147441864, -0.8690207600593567, -0.31487852334976196, -0.9617359638214111, -1.2740216255187988, -0.2833453416824341, -0.541366457939148, 1.5251373052597046, -1.3023327589035034, -0.6939727663993835, -0.19962993264198303, 1.7694289684295654, -1.6967462301254272, 1.3681888580322266, 0.2806156277656555, -0.36119934916496277, 0.595317542552948, 0.07390997558832169, 0.7511612772941589, 0.3364451825618744, 0.5157355666160583, 1.1303871870040894, 1.5082803964614868, -0.7137176990509033, -1.427573800086975, -0.2170180380344391, -1.532160758972168, 0.17250816524028778, -0.48270878195762634, 2.5228068828582764, -0.370777428150177, -0.8562456965446472, 0.40666884183883667, -1.0449433326721191, -1.4912704229354858, -0.08289913833141327, -0.22893939912319183, 1.1882392168045044, 0.7420518398284912, 0.03951285779476166, 0.31196483969688416, -0.1716698408126831, -1.9129555225372314, -0.29923757910728455, 0.11085177212953568, -0.5952096581459045, -0.05215977877378464, -0.3280794620513916, 0.5353677868843079, -0.17228323221206665, 0.7228249311447144, 0.6413356065750122, -0.30891209840774536, -2.388888359069824, -0.6092492938041687, -0.32018589973449707, 0.8564531207084656, 0.4416126012802124, -1.916532278060913, -1.5841445922851562, 0.548978865146637, -0.5754148960113525, 0.7193329930305481, 0.1144460141658783, -0.8091217875480652, -0.929633617401123, 0.9082798361778259, 0.5997425317764282, -0.09637364745140076, -0.3561246395111084, -2.2780630588531494, 2.2714686393737793, -0.7985108494758606, 1.1739509105682373, -1.0503329038619995, 0.40621712803840637, -0.9004265069961548, 2.8152737617492676, 1.538516640663147, -1.2966203689575195, -1.3118319511413574, -0.8853611350059509, -1.1857465505599976, 1.5701748132705688, 0.2627822756767273, 0.7983341217041016, 0.04072273522615433, 0.6144717335700989, -2.3100056648254395, 0.3761817216873169, -0.0015942519530653954, -2.1896088123321533, -0.6403033137321472, 0.4812735617160797, -1.8689985275268555, 0.2833506464958191, 0.3740381598472595, 0.27916255593299866, 0.293149471282959, -0.07475638389587402, 1.6748170852661133, 1.5631605386734009, -0.6561753153800964, 2.6562564373016357, -0.3538496494293213, 0.7305160760879517, -0.8235551714897156, 1.5264462232589722, 0.2910503149032593, -1.9357869625091553, -0.16072902083396912, -0.7329962849617004, -0.01186664029955864, -0.5549825429916382, -0.4342746138572693, -1.9902571439743042, 0.2590865194797516, -0.56367027759552, 0.8852447271347046, 0.8568508625030518, -1.7681281566619873, -0.9544835090637207, 1.1056842803955078, -1.3125278949737549, -0.5821254849433899, -2.0889339447021484, 0.06948617845773697, 1.1228911876678467, 1.240514874458313, -0.9878027439117432, 0.027396179735660553, 0.01744617521762848, -0.6584962010383606, -0.7847990989685059, -0.06891006976366043, 0.5340245962142944, -1.1873977184295654, 0.2414330691099167, 0.660631537437439, -0.20597273111343384, -0.7292153239250183, 0.3756501078605652, -0.46197834610939026, 0.40620559453964233, 0.5036779046058655, 1.5085257291793823, 1.4088141918182373, 0.6884900331497192, 0.21128469705581665, 0.324885755777359, -0.7529796361923218, 1.005077600479126, -0.42973291873931885, -2.3466968536376953, -1.8694883584976196, 0.9162473678588867, -0.5616573095321655, -1.0686161518096924, 1.811269760131836, 1.2836872339248657, -0.825727641582489, -0.5609239339828491, 1.096627116203308, -0.00027947514900006354, 0.36534368991851807, -0.5078763365745544, -1.11483895778656, -1.0088536739349365, 0.5765669941902161, 0.4193894565105438, 1.1722192764282227, 0.5439154505729675, -0.2590302526950836, 1.5126124620437622, -1.217869758605957, -0.20615442097187042, 0.5657449960708618, 1.3502494096755981, -0.3667427897453308, -0.5240980982780457, 1.2750215530395508, 0.09987432509660721, 0.543273389339447, -0.7576220035552979, -0.25775089859962463, 0.19597861170768738, -1.049694538116455, -0.08201682567596436, -0.033700112253427505, -1.6530547142028809, -0.9349321722984314, -2.3204855918884277, 2.6754953861236572, -0.402233362197876, -0.09700911492109299, 2.399848222732544, 1.1847593784332275, -1.0288125276565552, 0.37270861864089966, 0.21244122087955475, -1.2748546600341797, 0.30352163314819336, -0.797181248664856, -0.8309336304664612, -0.21551892161369324, -0.3933257460594177, 0.9762027859687805, -0.40856364369392395, 0.3612614870071411, 2.4125351905822754, 0.6901980042457581, 2.6790316104888916, -1.2654039859771729, 0.6922515630722046, -0.21569229662418365, 0.36425796151161194, -0.8600760698318481, -0.1476176530122757, -0.6912891268730164, 1.0717748403549194, 1.5306590795516968, 0.5555070042610168, 1.2782238721847534, -0.010740482248365879, -0.44067832827568054, 0.2912063002586365, 0.5351722836494446, -0.7216646671295166, -0.15132446587085724, -1.7488240003585815, -0.3269572854042053, 2.5865888595581055, -1.0318975448608398, 0.8912322521209717, -1.2189388275146484, 0.17328470945358276, 0.4671213626861572, -1.3822143077850342, -0.8588706254959106, 0.41568732261657715, -0.15343008935451508, -0.9690290093421936, 2.720949411392212, 0.9132312536239624, 1.8510242700576782, -0.03173888847231865, -0.9520450234413147, -1.0011086463928223, -2.3190832138061523, -1.8804540634155273, -0.07676421850919724, -1.049027681350708, -0.28529849648475647, -0.49398860335350037, -1.326203465461731, -0.7412601709365845, 0.7035202980041504, 1.8467501401901245, 2.2626078128814697, 1.3207600116729736, 1.1142650842666626, 0.6135550141334534, 0.36737534403800964, 0.611476719379425, 0.38635867834091187, 0.21569645404815674, -0.8182148337364197, -0.22012318670749664, -0.8345444202423096, 0.23727752268314362, 0.4589307904243469, 0.40371736884117126, -0.04046687111258507, -1.1802802085876465, 0.0416034460067749, 0.67010098695755, 0.0644000843167305, 0.7283274531364441, -2.0198681354522705, 0.10798975825309753, -1.178847312927246, -0.06948286294937134, -0.28142666816711426, 0.9554446339607239, 1.4363962411880493, 0.12856833636760712, -0.932206928730011, 0.4598630368709564, -0.5013791918754578, -0.6442296504974365, -0.5805988907814026, -0.01354883424937725, -1.0884618759155273, 2.281033515930176, -0.7376071810722351, -1.1573295593261719, -0.7158625721931458, -0.844098687171936, 1.359187126159668, -0.05445040389895439, 1.068882942199707, 1.5338020324707031, 1.4529130458831787, 1.6661016941070557, 0.14904209971427917, 0.050360649824142456, 1.3040449619293213, -1.8656220436096191, 2.3476977348327637, -0.817341685295105, 1.0964854955673218, -1.4564265012741089, -1.072815179824829, -0.07406482100486755, 0.8768801093101501, -0.9844282865524292, -0.9007307291030884, -0.6068214178085327, 0.2720341384410858, -1.2229762077331543, -0.1256582885980606, -0.35599496960639954, 1.2363160848617554, 0.29969775676727295, -1.132279634475708, -0.35544031858444214, 0.7139960527420044, -0.4897811710834503, -0.5850315093994141, 1.02536141872406, 1.3225256204605103, -0.05958190932869911, 0.883571982383728, -0.04144150763750076, 1.8593947887420654, -0.7073067426681519, -0.3185592293739319, 0.8240015506744385, 2.277945041656494, -0.03937353938817978, -0.6234229207038879, 1.5078588724136353, 0.24910768866539001, -1.2251445055007935, 1.859060525894165, 0.7176626920700073, 0.06769958883523941, 0.36331501603126526, -1.1622889041900635, 0.9118862748146057, -1.3270671367645264, 0.29356512427330017, 0.4466720521450043, 1.5269657373428345, -1.8031636476516724, -0.9335216879844666, -1.2636775970458984, 0.02790542133152485, -1.6181517839431763, 0.14011649787425995, -0.40695488452911377, 1.4186553955078125, -0.7468497157096863, -1.23312246799469, -0.1408950686454773, -0.6512559652328491, -0.959373950958252, -0.7783911824226379, 0.09026484936475754, -0.48152413964271545, -0.6529598236083984, 1.5283604860305786, -0.6333922147750854, -1.7869305610656738, 0.6972005367279053, 1.4175329208374023, 0.5051308274269104, 0.6584421992301941, 0.5694683194160461, 0.3761494755744934, -0.11791706085205078, -1.0827792882919312, -1.3137409687042236, -0.5466960072517395, -0.150151327252388, -0.6736193299293518, -0.6176172494888306, 0.7311577796936035, 0.6554322242736816, -1.526865839958191, 0.09231607615947723, -0.47897058725357056, -0.24681168794631958, -2.04469633102417, -1.0594795942306519, 1.2846356630325317, 0.033231936395168304, 0.29689306020736694, 0.19803351163864136, -0.24683719873428345, 0.7895269989967346, 0.28159114718437195, -1.3757948875427246, 0.4142877459526062, 0.0705590471625328, -0.5275715589523315, 0.038992658257484436, 0.01774825155735016, 0.2822437882423401, 1.8249081373214722, -2.1340956687927246, 1.4165891408920288, 0.7150045037269592, -0.7898377776145935, -0.30886274576187134, -0.4984927475452423, 1.4391999244689941, -0.31939980387687683, 0.7967429757118225, -0.495953768491745, -1.3447765111923218, 0.4743714928627014, 0.6630711555480957, 1.044578194618225, -0.5757572054862976, 0.7161135077476501, -1.2549265623092651, -0.6137616634368896, -1.1399457454681396, -1.70242178440094, -2.075314521789551, -0.41577184200286865, -0.7085777521133423, -0.8314048647880554, -0.372638463973999, 1.1916736364364624, 1.2278919219970703, -0.20734591782093048, 1.3987314701080322, 0.44261276721954346, 0.47585779428482056, 1.6742589473724365, 0.6533206105232239, -2.7056877613067627, 0.7745642066001892, -0.4281489849090576, -0.9728554487228394, -0.21092520654201508, -2.091198205947876, -1.2292448282241821, 0.12697814404964447, -0.2703911066055298, -2.111262798309326, 0.2869761288166046, -0.13989481329917908, 0.4748460352420807, 1.1950712203979492, 0.1227942705154419, 0.8010343313217163, 1.090636134147644, -0.5051906108856201, -0.9953832626342773, -0.4764445126056671, -1.1868038177490234, 1.4308902025222778, -1.4588969945907593, 0.7909620404243469, -0.07134656608104706, 0.48677584528923035, -0.2935771942138672, 0.24013008177280426, 0.6836292147636414, -0.4007451832294464, -0.831100344657898, 2.5817832946777344, 0.5988742709159851, 0.2881444990634918, 0.5095055103302002, -0.0484679639339447, 0.0029293587431311607, 0.06619089096784592, 0.9076719880104065, 0.17640092968940735, -0.05528935045003891, -2.032224416732788, 0.3045627474784851, -0.4866114854812622, -1.5632890462875366, 0.5710713267326355, 0.36305373907089233, -1.4360307455062866, 0.3860135078430176, -0.8245120644569397, -0.10796481370925903, 1.3638293743133545, 0.7705707550048828, -1.100095510482788, 0.26307931542396545, -0.08851038664579391, -0.5732755064964294, 1.8827776908874512, 0.007224089000374079, 0.8438155055046082, -0.4607200622558594, 0.8113778233528137, -0.07361994683742523, 0.08733915537595749, -0.9561244249343872, 0.7740645408630371, -0.039913494139909744, 0.8901565670967102, -0.8313198685646057, -0.16916055977344513, 1.2767555713653564, 1.1002622842788696, -2.1885645389556885, -2.065472364425659, 0.8142979741096497, -1.993935465812683, -0.40242597460746765, 1.4876856803894043, -2.7341294288635254, 1.2824223041534424, 0.37827789783477783, -0.5137205719947815, -1.3335603475570679, 0.008750559762120247, -0.07726331800222397, 1.0017019510269165, 0.1325729787349701, 1.1717993021011353, 0.3045932650566101, -0.338218092918396, 0.366556853055954, 0.020448068156838417, 0.049056459218263626, 0.5932142734527588, 1.1170514822006226, 0.9275211691856384, -2.5234973430633545, -0.7757125496864319, 0.051605474203825, 0.796912670135498, -0.46245720982551575, -1.7182494401931763, 1.779441475868225, -0.32007065415382385, 1.1094887256622314, 1.3205478191375732, -0.3344782292842865, -0.7367470264434814, -1.9556283950805664, -1.4561694860458374, -0.07566429674625397, -1.0237939357757568, 0.42323434352874756, -0.605861485004425, 0.13445942103862762, 0.3063453435897827, -0.7575640082359314, -1.2699010372161865, 1.7236677408218384, 0.12874813377857208, -2.6613516807556152, -2.00982666015625, -0.9458010792732239, -0.9543848037719727, 0.22189123928546906, 1.0692241191864014, 0.18925929069519043, -0.9751620292663574, -0.9387046694755554, -0.7830458879470825, 0.7600299119949341, -2.0584375858306885, 0.9534734487533569, 0.6804428100585938, -1.413849949836731, 2.301985502243042, -1.0604989528656006, -0.3488839566707611, -0.26576146483421326, -0.28746190667152405, 0.015557804144918919, 0.24899660050868988, 1.5532512664794922, -1.077508568763733, 0.3330785632133484, -0.4538654386997223, -0.8740196824073792, -0.23322701454162598, -0.6045381426811218, -1.6868668794631958, 1.590168833732605, 0.04380318894982338, 1.757405400276184, -0.5522486567497253, -0.4656831920146942, 0.15886500477790833, -1.8621752262115479, -0.613641619682312, 0.20725221931934357, -0.07140980660915375, -0.1910376399755478, -1.798251748085022, -0.038747623562812805, -1.440334439277649, 0.47675439715385437, 0.20370306074619293, 1.1147735118865967, 1.1329606771469116, 2.2934393882751465, -0.9592466950416565, 0.5383080244064331, 0.6689094305038452, 0.551062822341919, 0.2975143790245056, -1.1097491979599, 0.46185043454170227, -0.44428977370262146, -0.8834711313247681, 0.5109003186225891, 1.5170435905456543, -1.4029139280319214, 1.8587901592254639, -1.6850016117095947, -0.9888293147087097, 0.6170415282249451, -0.19567160308361053, -0.012532563880085945, 0.17388834059238434, 0.13679386675357819, 0.6812220215797424, 0.4387129247188568, -0.1662527471780777, 0.9455811381340027, 0.2944285571575165, -0.7423739433288574, -0.9050125479698181, 0.9566770792007446, 0.658865213394165, -0.35682567954063416, -0.7576345801353455, -1.4819449186325073, 1.6738070249557495, 1.073194980621338, -1.0409314632415771, 0.08269354701042175, -0.7231172323226929, 0.06960524618625641, -0.2647290527820587, 0.33171457052230835, -0.26197493076324463, -1.5237199068069458, 2.27530574798584, -0.2489486038684845, -0.7717456221580505, 0.6437634825706482, 0.9163031578063965, -1.4407950639724731, -0.32947391271591187, 0.19509880244731903, 0.7193442583084106, 1.6515111923217773, -1.9534528255462646, -0.4893309473991394, -0.4321247637271881, 0.6261506080627441, 0.9299589991569519, -1.7804267406463623, -0.8661034107208252, -0.5896090269088745, -0.17938773334026337, 0.8615415692329407, -1.8766323328018188, 0.2701452076435089, 0.3890567123889923, -0.2733343839645386, -0.014382347464561462, -0.2614193856716156, 0.453213095664978, 0.22520147264003754, -0.07676941901445389, -0.23459909856319427, 0.033258069306612015, -1.480415940284729, -0.7140454053878784, -0.098129041492939, 1.6727826595306396, 1.5013781785964966, 0.3793432414531708, 0.41681817173957825, 1.6479108333587646, -1.2842260599136353, -0.20519918203353882, -0.34394219517707825, -0.2551940977573395, -2.166900873184204, -0.32731953263282776, -0.5212927460670471, 0.1921924352645874, 1.6977133750915527, 0.04711295664310455, 0.28971067070961, 1.9459973573684692, -1.2924443483352661, -1.0621930360794067, -0.3550087511539459, -1.5404860973358154, -0.2370338886976242, 0.5747765302658081, 1.2100703716278076, 0.4430985748767853, -0.9227020144462585, 0.39344170689582825, -0.3927694261074066, -1.9033902883529663, 1.5049803256988525, 0.6046195030212402, 1.027483344078064, 0.2803328335285187, 0.6133807301521301, 1.7705802917480469, -0.25289180874824524, 1.0402283668518066, 1.227104902267456, 1.1003555059432983, 0.5969096422195435, -0.7839598655700684, 0.1233217716217041, 0.13386036455631256, 0.05936010554432869, 0.572446882724762, -0.7348098158836365, -0.5265774130821228, 0.27901312708854675, 2.4851291179656982, 0.4935830235481262, 1.109969139099121, -0.6465469002723694, -0.7212985754013062, -0.10412567853927612, -0.6046353578567505, -0.8511051535606384, -1.5546139478683472, 1.202968955039978, -0.3150593936443329, 0.09788496792316437, -1.0096619129180908, 0.396157443523407, -1.4161118268966675, -0.724926769733429, 1.2375484704971313, -1.6515789031982422, 1.3493632078170776, -0.4573608338832855, 0.48249274492263794, -0.09759607166051865, 0.15406569838523865, -1.7329494953155518, 1.4751794338226318, 0.0707014948129654, 1.1937618255615234, -1.8290693759918213, 0.8235592842102051, 0.2652382552623749, 0.06322243809700012, 0.00622737780213356, 1.14131760597229, -0.9657149314880371, 1.0198330879211426, 0.4796029329299927, 0.22438883781433105, -0.3011690080165863, -1.5205347537994385, -0.6910306811332703, -1.0794243812561035, -0.5753669142723083, -0.18499043583869934, -1.27602219581604, 0.03922343626618385, 0.8144301772117615, 0.301553338766098, 0.003197726560756564, -0.8274069428443909, -1.0105689764022827, 1.4783862829208374, -0.23339273035526276, -0.6564840078353882, 1.0156577825546265, 0.7977996468544006, -0.8620297908782959, -1.5767946243286133, -2.233415365219116, -1.4436999559402466, -0.5655744075775146, -1.1627024412155151, -0.4011974632740021, 1.0041794776916504, -1.007077932357788, -0.14911870658397675, -0.8516157865524292, -0.862427294254303, -0.8243848085403442, 0.01644165813922882, 0.7521071434020996, 0.8218787908554077, -0.889546811580658, 0.8304575681686401, 0.7279824614524841, 0.0436549112200737, 0.9640882611274719, 0.6442086100578308, -0.464763879776001, -0.8507639169692993, -0.14934909343719482, 2.2366135120391846, 0.8722677826881409, -1.2117276191711426, 1.5370104312896729, 0.1190866008400917, 1.0994576215744019, 1.0490044355392456, 0.41989436745643616, -1.864743947982788, -0.1709272861480713, -1.077372431755066, -1.0637370347976685, 0.5457354187965393, 1.8800374269485474, -0.8976832628250122, 0.6545606851577759], "dtype": "float32"}, 1.0, 1.0], "e": {"__nd__": [-0.5295184850692749, 2.754258871078491, 5.560701370239258, 0.8539462089538574, -4.015494346618652, 1.2248015403747559, -10.273884773254395, -1.7849712371826172, 0.3368276357650757, -1.808152675628662, 3.4861502647399902, -1.9708752632141113, 2.0675127506256104, -2.7777364253997803, 0.5646079182624817, -1.5834251642227173, -8.523219108581543, -4.201859474182129, 0.4673463702201843, 6.506747722625732, -6.574802398681641, -0.19594484567642212, -3.965742826461792, -0.8482413291931152, -2.3110971450805664, -5.537709712982178, -1.165229320526123, -1.3378876447677612, -0.0937216579914093, 0.96894371509552, -4.190047264099121, 3.9027392864227295, -2.030001640319824, -2.133910655975342, 0.7249996662139893, -5.389819622039795, 0.9112109541893005, -1.4445974826812744, -7.997964382171631, 4.414233207702637, 2.547276735305786, 0.8481393456459045, 10.037147521972656, 0.739002525806427, -2.199859857559204, 5.908929347991943, -2.4636666774749756, 3.0194320678710938, 3.115605115890503, 4.578479766845703, 3.249335765838623, -0.4271836280822754, 1.0026137828826904, 1.817684292793274, -1.0569008588790894, 7.329136848449707, -3.0672006607055664, -0.5628576278686523, 6.015835762023926, 0.22090508043766022, 3.381605863571167, -1.4865050315856934, -3.9004759788513184, -1.9037015438079834, -3.193267345428467, 1.8873591423034668, 0.49534422159194946, -1.0125221014022827, 7.236657619476318, -5.806227207183838, 4.239978790283203, 3.1903655529022217, -8.053297996520996, 0.9941644668579102, 4.264833450317383, -5.349156379699707, -2.2416863441467285, -1.6646395921707153, -6.119670867919922, -0.534421980381012, -8.097407341003418, -4.5032196044921875, 2.0196194648742676, -2.048205852508545, -3.61566424369812, 1.7166953086853027, 6.9015021324157715, -4.299537658691406, 3.8031208515167236, -2.5863893032073975, 1.13722825050354, 1.1451852321624756, -3.223210334777832, 6.819242000579834, 1.9147027730941772, 7.131101608276367, 4.950374126434326, -0.18999943137168884, 4.875789165496826, -11.820843696594238, -0.3923841118812561, -2.425107479095459, 2.841655731201172, 0.6675935387611389, -0.1356518268585205, -2.411529064178467, -1.953477382659912, 4.678834438323975, -5.420405387878418, -4.975705623626709, 4.471780776977539, -1.2502752542495728, -3.0518808364868164, -1.4537582397460938, -7.318404197692871, -3.8111233711242676, 3.5480892658233643, -1.2289217710494995, -1.9209728240966797, 2.1906676292419434, -4.670973777770996, 1.5897918939590454, 3.274451971054077, -3.6680784225463867, -4.337606906890869, 1.3455113172531128, 4.937046051025391, -1.320927619934082, 0.9537494778633118, -1.6206787824630737, 1.3285428285598755, 2.6223888397216797, -1.3988102674484253, 8.045259475708008, -2.870572328567505, -3.8697144985198975, -4.922560691833496, -0.7248091697692871, -0.6225161552429199, -7.89783239364624, -1.343231201171875, 2.186572790145874, 4.421361446380615, 0.8615562915802002, -3.549586534500122, 3.7517666816711426, -2.219996690750122, 1.985753059387207, 1.371921181678772, -4.348838806152344, 3.6916792392730713, 3.7130327224731445, -0.6755874752998352, -2.201974391937256, -4.026535511016846, 5.4433135986328125, -3.1927106380462646, 2.177459239959717, 0.1673775315284729, -1.2620792388916016, 4.3790459632873535, -2.569936513900757, -1.3140757083892822, -2.348794937133789, 3.2791357040405273, -0.39185631275177, 5.3409624099731445, 2.979079008102417, 2.4348340034484863, -0.07797767966985703, 2.475398063659668, 1.5116567611694336, -1.986895203590393, 1.1362807750701904, 2.401663064956665, -1.5887409448623657, -0.32929888367652893, 2.966279983520508, 2.4966375827789307, 0.6414757370948792, 1.0288856029510498, 10.28821086883545, 3.7460877895355225, 3.655809164047241, -1.7079782485961914, 4.403083324432373, 2.7907297611236572, -3.635913610458374, 4.172468185424805, -6.165612697601318, -0.8805257081985474, 1.8306348323822021, -2.9756243228912354, -4.346614837646484, -3.406146287918091, 3.458653688430786, -3.7889161109924316, 7.306251525878906, -1.5639302730560303, -1.97645103931427, 3.8831443786621094, -1.9943095445632935, 2.1607418060302734, -2.0478901863098145, 4.194332599639893, -1.2609399557113647, 5.291173934936523, -0.1059546172618866, -1.4665839672088623, -0.9473287463188171, -4.456921577453613, 2.080286979675293, 5.23103141784668, -1.5798896551132202, -3.393399238586426, -7.580411911010742, 1.4832590818405151, -3.599891424179077, 5.7813720703125, 0.6817144155502319, -2.862212657928467, -1.0873483419418335, -1.5733448266983032, -9.137618064880371, 1.6008671522140503, 4.964025497436523, 1.1911646127700806, -2.3147542476654053, 5.934385776519775, -5.681399822235107, -5.985605716705322, 2.0679562091827393, 2.7999930381774902, 3.3344953060150146, 0.31306710839271545, -0.07437475770711899, 4.3979597091674805, 3.5102083683013916, 0.17767880856990814, 4.910040378570557, -2.871485471725464, -1.1487865447998047, -4.380099773406982, -2.8152291774749756, -1.756582498550415, 8.064868927001953, 1.6387684345245361, -4.569246768951416, -3.035088062286377, -3.9107930660247803, 1.750648021697998, 1.1989632844924927, 5.890310764312744, 1.1048994064331055, 4.607927322387695, 4.958552837371826, -3.1933860778808594, 2.4604105949401855, 0.29482799768447876, -5.578596591949463, 0.33263716101646423, 7.040124416351318, -2.3700413703918457, 5.359157562255859, -5.340386390686035, 4.7980217933654785, -2.7157366275787354, -1.3096810579299927, 3.8770225048065186, 0.2004374861717224, 2.845081090927124, 4.902618885040283, 1.401360273361206, 1.733751893043518, -0.2627426087856293, 3.836770534515381, -2.9854514598846436, 5.404202461242676, 6.018944263458252, -1.4025943279266357, -1.0227717161178589, -3.576004981994629, -4.964715480804443, -9.204568862915039, 2.7343850135803223, -5.212189674377441, 6.285083770751953, 0.6433649659156799, -5.642700672149658, -7.1550493240356445, 3.1635398864746094, -6.865851879119873, -2.1888344287872314, -1.0278539657592773, 2.049182176589966, -0.3822205364704132, 4.7374749183654785, -5.776726722717285, 1.4461368322372437, 2.6343834400177, -0.9071314334869385, -2.0352976322174072, -6.077696323394775, -4.256430625915527, -1.081869125366211, -2.823748826980591, 0.12020616233348846, 4.943065643310547, -0.21526212990283966, 0.5059369206428528, 0.07274657487869263, 1.6626911163330078, -0.5157811045646667, -1.785239338874817, -1.2278038263320923, 1.678447961807251, 0.8407811522483826, -0.8745247721672058, 0.6158670783042908, -1.5495530366897583, 2.172877311706543, 4.168057918548584, -5.3039679527282715, 0.5790998935699463, -6.205977916717529, -3.8873114585876465, 6.662840366363525, 2.1926233768463135, 1.275054931640625, -7.0882744789123535, 0.815606415271759, -0.7701076865196228, -1.5410089492797852, -2.9587759971618652, 0.09018663316965103, -8.954025268554688, -4.442501068115234, 1.2805784940719604, 3.0676565170288086, 4.083044052124023, 3.6494362354278564, 3.5270578861236572, -0.3597830832004547, 1.8329905271530151, 3.65163516998291, 5.906367778778076, 0.7117679715156555, -2.6979644298553467, 2.130361318588257, 1.8557355403900146, -0.9463557600975037, -0.28130513429641724, -4.417299747467041, -2.729116916656494, -1.2832435369491577, -1.4054149389266968, -1.6130173206329346, 2.6809589862823486, -0.5048484802246094, -1.7397209405899048, 3.2247965335845947, 4.40640115737915, -1.91133713722229, 4.847587585449219, 1.660672903060913, -1.2515143156051636, 0.11440971493721008, 5.8902974128723145, -4.653350830078125, 5.673423767089844, -1.0598535537719727, 7.153740406036377, 0.14203941822052002, 5.08027458190918, -5.131679058074951, -0.7268857955932617, -1.52354896068573, 1.8720471858978271, 3.6911449432373047, 8.34604549407959, 5.021758556365967, 3.204385757446289, -5.543745040893555, -4.310609817504883, 4.076088905334473, -0.4926287531852722, -3.772552967071533, -0.25019770860671997, -4.629889011383057, -3.493540048599243, 2.2105722427368164, 5.430585861206055, 1.785426378250122, 2.1245839595794678, 6.968151569366455, 9.219965934753418, -0.07706894725561142, -1.6292320489883423, -1.0003013610839844, 0.0073570855893194675, -3.49487042427063, 2.3650963306427, -1.1815718412399292, -1.2752922773361206, 0.01953296735882759, 3.012183427810669, -5.749299049377441, -0.7018262147903442, -2.7707364559173584, 1.750906229019165, 1.418001651763916, -1.2221955060958862, 1.0598474740982056, -7.757817268371582, 0.2454855740070343, 0.30122479796409607, -0.8201799988746643, -5.633844375610352, -6.4377875328063965, -0.5814494490623474, 1.4102147817611694, -5.5110907554626465, -3.299286127090454, 0.23558764159679413, -3.152446746826172, -1.6427643299102783, 5.450442790985107, 0.6766123175621033, 3.5019938945770264, -2.2872114181518555, -0.8107966780662537, 3.7912776470184326, -4.158215045928955, -6.978079319000244, -2.017824172973633, 0.6601020693778992, -3.0133841037750244, -2.110537528991699, -1.2643377780914307, 4.001981735229492, 4.2138261795043945, 8.184996604919434, -1.8900024890899658, 1.8234361410140991, 1.5969195365905762, 3.8853492736816406, 0.9045496582984924, -1.3680459260940552, 4.059579372406006, -1.060347557067871, -4.846780776977539, -0.3161216080188751, 2.659224510192871, -2.313908815383911, 3.2558515071868896, -7.020528316497803, 3.4145867824554443, -0.8177712559700012, 2.1395528316497803, 1.7822171449661255, 4.553943157196045, 0.2611539363861084, 4.560637474060059, -3.366786479949951, 6.0954389572143555, 1.6752657890319824, 1.583977222442627, -4.37927770614624, -2.2186741828918457, -2.1879122257232666, 4.043248176574707, -4.72072696685791, -2.2788281440734863, -0.07892880588769913, 1.616390585899353, 0.11439740657806396, 5.373762607574463, 1.8036489486694336, 6.20374059677124, 4.35078239440918, -3.8529651165008545, 3.2403528690338135, 7.24912691116333, 4.337447643280029, 2.663576126098633, 0.9598519206047058, 0.7110373377799988, -4.161168575286865, -2.662630558013916, 4.96030330657959, 9.407334327697754, -2.4926791191101074, -7.505834102630615, 7.735474586486816, -4.707373142242432, 2.728991985321045, 1.3492028713226318, -3.2089548110961914, 0.20196720957756042, -1.9142805337905884, 1.707857370376587, -1.881907343864441, -5.98701810836792, 2.556899309158325, -2.5412516593933105, 3.1765270233154297, 4.250544548034668, -3.091427803039551, -0.027465375140309334, -0.34795305132865906, 0.35705479979515076, -0.7021280527114868, 4.5380940437316895, -1.5499516725540161, -2.3411669731140137, -3.9412405490875244, -3.455322742462158, -3.6630566120147705, -1.0530685186386108, 2.328472852706909, 0.5276951193809509, 6.956138610839844, 3.063593626022339, -5.434713840484619, -3.79561448097229, -1.9523091316223145, -3.537949800491333, -1.5966547727584839, 1.6171720027923584, 5.264712810516357, 4.06089448928833, 1.909519910812378, 8.007857322692871, -6.970489978790283, 1.1760624647140503, 2.386796236038208, -2.022387981414795, 1.540103793144226, 1.3249366283416748, -6.0173234939575195, -3.2674732208251953, 0.8697202205657959, 4.601898670196533, -1.1196863651275635, 1.3093456029891968, 1.4353969097137451, -0.5757982134819031, 0.15871545672416687, 4.762365341186523, 1.9064406156539917, -3.690387725830078, 1.5349159240722656, -3.609598398208618, 4.496927738189697, 3.9113550186157227, 0.5300276875495911, 2.4078454971313477, 2.085587501525879, -2.5023887157440186, -2.497870683670044, 3.6939010620117188, 0.8081140518188477, 2.366314649581909, -0.30621033906936646, -1.5137536525726318, -1.5155589580535889, 3.8066132068634033, -0.9427905678749084, -1.876035213470459, 7.521900177001953, 1.8754867315292358, 1.505614995956421, 0.672037661075592, -0.8243383765220642, 2.840812921524048, -2.3043978214263916, -4.254391670227051, 3.4335379600524902, -0.028972303494811058, 4.278877258300781, -0.10339564830064774, -1.6336593627929688, 1.5686850547790527, -3.6018240451812744, 1.3676133155822754, 0.379176527261734, -1.1171537637710571, -0.07008086144924164, -3.7329134941101074, 0.7888626456260681, 2.434110641479492, 0.7607045769691467, -7.089207172393799, 2.284163236618042, 4.521142959594727, -3.9108848571777344, 1.7825803756713867, -3.726592779159546, -5.686978340148926, -3.434685468673706, -2.9537248611450195, -0.5844587683677673, 4.034427642822266, 7.835236549377441, 3.3929667472839355, -1.4633272886276245, 1.5002169609069824, -1.088330864906311, -8.796090126037598, 2.4208524227142334, -3.082357406616211, -0.45048463344573975, -2.177527904510498, 1.512113094329834, -0.5428362488746643, 2.923948049545288, -4.9090142250061035, 0.6994421482086182, -0.30801257491111755, -1.6608649492263794, -1.9960812330245972, 0.337848424911499, -5.238579750061035, 0.31890133023262024, 5.1009416580200195, 3.848681688308716, -7.279890537261963, -1.0264440774917603, -1.1626989841461182, 3.666698694229126, 5.730564594268799, -4.83839225769043, -2.2148282527923584, -2.082671880722046, -1.1989717483520508, 3.8224129676818848, -0.03071976639330387, -1.1645939350128174, -1.9782485961914062, 2.451559066772461, -1.4346363544464111, -0.02768162079155445, 4.589480876922607, 2.749608278274536, 0.4063800275325775, 3.3339908123016357, -2.05609393119812, -4.0286664962768555, 0.26701268553733826, 5.022377014160156, 5.7098069190979, -0.5881827473640442, -3.780724287033081, 0.6198804974555969, -4.716238498687744, -2.642627239227295, 3.1002297401428223, -2.676422357559204, -2.391180992126465, -1.9150168895721436, -6.090802192687988, 0.16021570563316345, 3.9715871810913086, 0.7437897324562073, -8.05097770690918, 2.1083593368530273, -6.602109909057617, -7.075253963470459, 1.3979555368423462, -5.134024143218994, 5.74266242980957, 0.5627196431159973, -2.9941766262054443, -1.4123218059539795, -2.5573854446411133, -4.8981122970581055, -1.2455631494522095, -5.355705738067627, 2.4980270862579346, 7.667332172393799, 5.889838695526123, 4.121339797973633, 2.9983937740325928, -3.5292317867279053, -0.7129350304603577, -0.347741961479187, -2.4730658531188965, -8.744186401367188, 2.3955352306365967, -3.910684585571289, -0.3423275351524353, 1.7886112928390503, 1.8278151750564575, 0.8100950121879578, 0.5831413865089417, 1.3763988018035889, 2.3149254322052, -0.7541993856430054, -4.256141185760498, 2.0030322074890137, 7.872669219970703, 0.1921457201242447, 5.368526935577393, -3.8228933811187744, 1.3995760679244995, 8.654876708984375, -5.2746477127075195, -4.896797180175781, 2.285135507583618, 3.0336689949035645, 3.2713663578033447, -5.343298435211182, -1.368135929107666, 8.237601280212402, 3.5748214721679688, -0.36600661277770996, 4.874340057373047, -2.6757326126098633, 0.031062442809343338, -7.056341171264648, 0.9146508574485779, -3.5892906188964844, -0.6979122757911682, 0.6053599119186401, -3.197723865509033, -1.1640217304229736, 5.366218090057373, -5.575213432312012, -2.32983136177063, -0.35159531235694885, -1.1117899417877197, -0.8621717691421509, -4.206759929656982, -6.264226913452148, -1.0599122047424316, -1.4044402837753296, 6.374999046325684, -1.1050184965133667, 0.3136017620563507, -2.7430191040039062, -4.230371475219727, -2.576836585998535, -3.9709692001342773, -2.6357452869415283, 3.987804651260376, -4.042391300201416, -0.5266752243041992, 1.8338690996170044, -4.11877965927124, -10.586953163146973, -1.1562482118606567, 1.1234842538833618, 2.134307622909546, -1.4562602043151855, -1.693381905555725, 2.5664055347442627, 2.496983289718628, -2.6111936569213867, 0.0011361378710716963, -2.2595841884613037, -4.19318151473999, -0.7895233035087585, 2.348691701889038, 7.02818489074707, -5.901005268096924, -1.4972105026245117, -1.6389552354812622, 2.296412467956543, -4.145069122314453, 1.6484583616256714, 3.870877504348755, -4.053528308868408, -9.180888175964355, -3.640378475189209, 1.9951237440109253, -3.0927817821502686, 2.581369400024414, -1.0761184692382812, 1.829458236694336, 5.0495991706848145, -7.840104103088379, 0.054895613342523575, 0.15082432329654694, 1.254054307937622, -2.122607707977295, -2.3963470458984375, 1.1308127641677856, -7.071045875549316, 2.599395751953125, 0.9946601390838623, -6.992441654205322, -1.372687578201294, 4.38127326965332, -2.4975948333740234, -1.326078176498413, 4.50386905670166, 0.38496437668800354, 1.5939182043075562, 0.616237223148346, 2.4278366565704346, -3.527379035949707, -2.327725887298584, -4.1132049560546875, 1.8537523746490479, -2.6787898540496826, -1.2656219005584717, -6.291840076446533, 0.7016156911849976, 3.040769100189209, 0.4052915573120117, 2.923496961593628, 6.941057205200195, -4.705995559692383, -0.5887739658355713, -5.057950973510742, 0.31587550044059753, -4.321071147918701, -5.67308235168457, 3.9531731605529785, 3.0078940391540527, -2.18045973777771, -5.488852024078369, 2.9352498054504395, 4.297744274139404, -1.7424379587173462, -1.4344338178634644, 1.4904272556304932, 1.9075162410736084, -1.8118547201156616, 4.032560348510742, 3.366657257080078, -1.5991021394729614, 7.3473711013793945, -2.8865902423858643, 4.995513439178467, -10.252765655517578, 1.3450572490692139, -6.820870399475098, -4.483686447143555, -0.7046402096748352, 0.9016947150230408, 0.4018546938896179, -4.64096212387085, 1.3863239288330078, 1.1548247337341309, 4.184823036193848, 4.942766189575195, -5.903769493103027, -3.0514910221099854, -2.4720771312713623, -3.564443826675415, -4.085399150848389, 4.45201301574707, 2.479706048965454, 0.8038346171379089, 3.4034183025360107, -1.1506493091583252, 8.03400993347168, 3.837846040725708, 2.351975202560425, 7.566680431365967, -1.1104496717453003, 0.277138888835907, -1.9009472131729126, -1.3333888053894043, -2.065420389175415, 1.4059418439865112, 0.7338278889656067, -1.647998332977295, 0.6383429765701294, 4.061895847320557, -2.4886584281921387, 13.697687149047852, 0.9530127048492432, 2.4220831394195557, -1.158345103263855, -4.10275936126709, -0.6309707760810852, 1.107243537902832, -0.634581446647644, 3.5630433559417725, -4.545112133026123, -4.2063775062561035, -0.4346005916595459, 1.8758631944656372, -0.5080443024635315, -1.5341192483901978, -7.873716831207275, 5.0835700035095215, 4.83737325668335, 5.044010639190674, -0.9007776379585266, -3.65313458442688, 1.4739439487457275, -1.6281394958496094, 9.451645851135254, 0.9434561729431152, 5.703021049499512, 0.5344127416610718, -0.18602117896080017, 5.805610179901123, -1.008358359336853, 2.6910457611083984, 4.962507724761963, -1.1087502241134644, -0.8073745369911194, -2.955048084259033, 1.980250597000122, -6.3116679191589355, -0.31696686148643494, -4.384702205657959, -0.6417598128318787, -6.0630292892456055, 6.471841812133789, -1.4002701044082642, -6.774587154388428, -6.695878505706787, 5.875709056854248, 6.444735527038574, -3.262016534805298, -3.4871792793273926, -0.6515277028083801, -2.0884034633636475, -0.17757472395896912, 3.226729393005371, -4.889808177947998, -5.3650641441345215, -0.9146294593811035, 5.275411128997803, -4.687684535980225, 1.0145527124404907, -2.2031545639038086, 0.7862878441810608, 2.1157076358795166, 5.791757106781006, 3.8659985065460205, -0.6496984362602234, -3.7522308826446533, -5.604254722595215, -1.7839936017990112, 2.09104323387146, -4.801455020904541, 7.80038595199585, -6.665782451629639, -7.441126823425293, 7.962265491485596, 1.7300138473510742, -5.552694797515869, -1.1190563440322876, 0.502668023109436, -1.021241545677185, 4.403118133544922, -7.381229400634766, 2.25002121925354, 5.319074630737305, 2.4213216304779053, 0.663562536239624, 3.3898327350616455, 1.084625482559204, 5.76566743850708, -1.242284893989563, -0.8330307602882385, 4.57180643081665, -0.6008824110031128, 3.9775660037994385, 1.9102281332015991, 0.48026910424232483, 2.2811832427978516, -0.4435824453830719, -6.201239109039307, 5.171464443206787, -1.5228724479675293, -1.3782238960266113, 3.1674273014068604, -0.24886347353458405, 0.6871916055679321, -3.4353277683258057, 1.5989705324172974, 2.1218507289886475, 0.34127286076545715, -2.580153226852417, -3.012737989425659, 1.918196439743042, 4.35426664352417, -1.1103589534759521, 1.0231835842132568, 2.105349540710449, -4.190128326416016, -3.1380319595336914, -0.6493158340454102, -1.045632243156433, 3.773181915283203, -0.5287072062492371, 0.8696373105049133, -3.4556875228881836, -2.558394432067871, 0.3856971263885498, 11.380101203918457, 5.954004764556885, -5.518918514251709], "dtype": "float32"}}, {"a": [{"__nd__": [[0.6523198485374451, -0.898112416267395, -0.8346267342567444, -0.552514910697937, -0.42580723762512207, 0.9370338320732117, 1.374843955039978, -1.4025840759277344, -0.09877060353755951, 0.28477177023887634, 1.3124338388442993, 0.5215944051742554, -0.780512809753418, -0.041624516248703, 1.2855017185211182, -0.20674020051956177, 0.21642112731933594, -1.28748619556427, -0.8245862126350403, -0.7063716053962708, -0.30012720823287964, 0.7165842056274414, -1.4346646070480347, 0.7080801129341125, -0.5877497792243958, 0.3255764842033386, -0.6510769724845886, 0.5115991234779358, -0.7216214537620544, 0.46308645606040955, -0.9923100471496582, -0.5779761075973511, -0.8761054277420044, -0.3858271837234497, -1.039548635482788, -0.6070653200149536, -0.8616291284561157, -0.3760157823562622, 1.1887160539627075, -0.055093951523303986, 0.9411140084266663, -1.278233289718628, 1.3358741998672485, -1.412827968597412, 0.12475715577602386, -0.24062208831310272, -0.25373682379722595, -0.6428217887878418, -1.601700782775879, -0.33806413412094116, 0.7758205533027649, 0.6243605613708496, 0.293040007352829, 0.7494502663612366, -0.18294930458068848, 2.4926111698150635, -0.4065890610218048, -1.2799177169799805, -1.4547795057296753, -1.3356597423553467, -0.08537273854017258, 1.568645715713501, -0.1946437805891037, -0.8167173266410828, 1.005753755569458, 0.7376976013183594, 2.140909194946289, 0.4967733919620514, -0.21094933152198792, -1.6599042415618896, 0.2669794261455536, -0.16796821355819702, -0.5517433881759644, -0.39707839488983154, -0.057712070643901825, -0.333829402923584, 0.469475120306015, 0.08801571279764175, 0.03561118617653847, -1.1864898204803467, -1.5979810953140259, -1.2686997652053833, 1.144747018814087, 1.6158974170684814, -0.380765438079834, 0.48379653692245483, -0.4139626622200012, 1.1052639484405518, 0.8540209531784058, 0.5472885370254517, -0.3017079532146454, -2.028714179992676, -0.7466735243797302, -0.8011590242385864, 0.6089674830436707, -0.2022332400083542, -0.7003514766693115, -0.41226232051849365, -3.7308218479156494, 0.6565721035003662, -0.3026498258113861, 1.4651819467544556, -1.1731849908828735, -1.2467842102050781, -0.11110871285200119, 0.2062329351902008, 1.2646902799606323, 0.5298155546188354, 0.5566960573196411, -0.9377148747444153, 0.08586713671684265, -1.2584431171417236, -0.5145003199577332, 0.3612375557422638, 0.04370392858982086, 0.37766626477241516, -0.5323612689971924, 0.41319534182548523, 1.6376327276229858, -0.07442846894264221, -0.44824740290641785, -0.8939467668533325, 0.8309603333473206, -2.040231704711914, -0.05755169317126274, -1.4208496809005737, 1.7700490951538086, -0.33515727519989014, 0.5519348382949829, 1.4999973773956299, 1.1244536638259888, 0.2681663930416107, 0.18918029963970184, -0.5004573464393616, -0.769028902053833, 0.6813511252403259, 0.2201065570116043, 0.028141742572188377, -1.563266634941101, -1.8176156282424927, -1.0223462581634521, -0.9836248159408569, 0.9801059365272522, -0.716320812702179, 0.6706400513648987, 0.8644378185272217, 1.3805434703826904, 0.36813122034072876, 1.4092583656311035, 0.8116883039474487, -1.8111920356750488, -0.08918630331754684, -0.6442034244537354, 2.1096014976501465, 1.0180234909057617, 0.0843425840139389, -0.051882658153772354, 0.4818551242351532, -0.18708078563213348, -1.2951407432556152, -0.5998444557189941, -1.0453159809112549, -0.015936534851789474, 0.32265332341194153, 0.8045724034309387, 0.8668615221977234, -1.2890362739562988, -1.1479837894439697, 1.906623363494873, -2.184835910797119, 3.061591386795044, 1.352884292602539, 1.4288227558135986, -0.33956480026245117, -0.6895110011100769, 0.9026100635528564, 0.30481451749801636, -0.7522941827774048, 0.6733617782592773, -0.26109710335731506, -0.7422232627868652, 1.6152013540267944, -0.8088045120239258, 0.5286970138549805, 1.1431642770767212, 1.3372281789779663, -1.2701668739318848, -1.0512006282806396, 2.0243778228759766, 0.46087542176246643, 0.5211114883422852, -0.7199891209602356, -2.8937063217163086, 0.0236374344676733, 0.8539227843284607, 0.20643962919712067, -0.08098966628313065, -0.23016692698001862, -1.6187303066253662, 0.033270079642534256, 0.6961609125137329, 1.0491547584533691, 0.3450075685977936, 0.17494802176952362, 0.21825112402439117, 0.5010176301002502, 0.15923191606998444, 0.2361951321363449, 2.007863759994507, -0.2368309497833252, 0.4912867844104767, -1.040194034576416, -0.0077346377074718475, -1.3093407154083252, -0.5865510702133179, 0.6705282330513, 0.43943557143211365, 0.17392346262931824, -0.07578988373279572, -1.033687710762024, -0.7670315504074097, 0.42609694600105286, 1.1928901672363281, -1.3451930284500122, 1.1419910192489624, -0.17620708048343658, 0.09814273566007614, 0.19591765105724335, -0.020749690011143684, -0.4031887948513031, -0.8944997191429138, -1.2332637310028076, -0.45713475346565247, 0.8432285189628601, -0.45536088943481445, 0.8967458009719849, -0.35716280341148376, 0.8463994264602661, 0.1271202564239502, -0.9155188202857971, -0.006967056542634964, 0.9758762121200562, 0.542049765586853, -0.34479427337646484, 0.3832947909832001, 0.08559457957744598, 0.62025386095047, -1.5813477039337158, 0.1556609719991684, -1.7139450311660767, 0.28171980381011963, 1.177071213722229, -0.44066885113716125, 0.1154659241437912, 0.538289487361908, -0.8477392792701721, -1.3780077695846558, -0.6271535754203796, -0.586148202419281, 0.44412732124328613, 0.7629350423812866, -0.3812073767185211, -0.2704865634441376, -0.1953340619802475, 0.8601904511451721, -0.019491085782647133, -0.38932231068611145, -1.056932806968689, -0.5937861800193787, 0.9547079801559448, -0.09891902655363083, -1.0599350929260254, 1.4875202178955078, -0.3683955669403076, -0.9892935156822205, 1.1525623798370361, 0.5539851188659668, 0.6481344103813171, -0.18874728679656982, 0.3476102650165558, 0.4073660671710968, -1.9360780715942383, 0.2758105397224426, -0.6598736643791199, -0.22412416338920593, 0.033196236938238144, -0.6694636344909668, -0.5828676819801331, -0.04660945385694504, -0.7432096600532532, -1.2028197050094604, -2.1064212322235107, 0.3103748559951782, 1.5198169946670532, -0.67316734790802, -1.2876709699630737, 0.365224152803421, -0.7016327381134033, 1.1368175745010376, -1.1943162679672241, 0.037058863788843155, 1.112297773361206, -1.28659188747406, 0.06894142925739288, -1.1979591846466064, 1.5910242795944214, -0.12817706167697906, 0.9363123774528503, 0.9229689240455627, 0.02651522122323513, -1.2404462099075317, -0.2376406192779541, -0.1947043389081955, -1.7012500762939453, -0.402604341506958, -0.6320072412490845, -2.175746440887451, 0.6845879554748535, -1.2423348426818848, -1.208176612854004, 1.6002055406570435, -0.1921270191669464, 1.0079618692398071, 2.584021806716919, -2.319920063018799, 0.7023690342903137, -0.8100431561470032, 0.47621312737464905, 0.6314881443977356, 0.4548593759536743, -0.05256062000989914, -0.605954110622406, -0.9585229754447937, 1.1539727449417114, 0.38136425614356995, 1.0385695695877075, -0.9409974813461304, -0.7569501399993896, 0.8462123274803162, -0.1625128984451294, -1.1228184700012207, -0.9493299126625061, -0.46894901990890503, -1.1969554424285889, 0.382553368806839, 0.7232791781425476, 0.4201492369174957, 0.4106355905532837, -0.9896339178085327, -0.8656824827194214, -0.1883632242679596, -0.5007064938545227, 0.7636376619338989, 1.2144712209701538, -0.9311436414718628, -1.7275477647781372, 0.6714220643043518, 1.7720589637756348, 0.17092318832874298, 0.8725302219390869, -0.5521227121353149, -0.4545115828514099, 0.7851091027259827, -0.6431713700294495, -0.16637347638607025, 0.3565385043621063, -0.3011041283607483, -1.200475811958313, -1.2899168729782104, -0.7265344858169556, 0.020271027460694313, 0.5615281462669373, -0.6153916120529175, 0.6198497414588928, 0.3460098206996918, -1.1184947490692139, -1.6111828088760376, -0.5489979982376099, -0.024207215756177902, 0.8159452676773071, 0.48573872447013855, -0.9089424014091492, 1.2382616996765137, -0.12352298200130463, -0.7820594310760498, -0.3440423905849457, -0.315022349357605, 0.5297443866729736, -0.4218564033508301, 0.7788152694702148, 0.5937626957893372, 0.8519986271858215, 0.9269646406173706, -0.13469719886779785, -3.106342077255249, 1.0381077527999878, 1.27083420753479, -1.169532060623169, -0.9713426828384399, 1.2978756427764893, 1.4832650423049927, 1.1376384496688843, -0.11655688285827637, -0.5628061890602112, -0.10793918371200562, 0.43139517307281494, -0.24887774884700775, 0.6191493272781372, 0.4333876967430115, 0.43421709537506104, 0.9743891954421997, -0.34027042984962463, -1.4789924621582031, -0.696315348148346, -0.18391114473342896, 1.0199733972549438, -0.6974698901176453, -1.4341732263565063, -0.09014058858156204, 3.2668402194976807, -0.8340840935707092, 0.6582422256469727, -2.3957254886627197, -0.848527729511261, 0.527066707611084, 0.487192839384079, -0.4976673424243927, 0.04614515230059624, 0.9912400841712952, -0.5477085113525391, -0.4814758002758026, -1.2958904504776, 0.8650277256965637, -0.9019257426261902, 0.6411287784576416, -0.17779690027236938, 0.26788774132728577, -0.05029958114027977, 0.2274087816476822, 0.23944708704948425, -0.7205982208251953, -0.12423193454742432, 0.7006639838218689, 1.434738278388977, -0.33326366543769836, 0.6359443068504333, -0.5893798470497131, -1.9042890071868896, -0.05177699774503708, 0.50518798828125, -1.046701192855835, -0.8086416125297546, -0.3285224735736847, -0.4552499055862427, 0.37825992703437805, 1.3518694639205933, 1.900270700454712, -0.6585808992385864, 1.9363675117492676, 1.2618595361709595, -1.359757900238037, -0.29193276166915894, -0.15950541198253632, -0.34566572308540344, -1.442223072052002, 0.9360767006874084, 1.6231898069381714, 0.17635364830493927, -0.2111005187034607, 1.8690128326416016, -1.232540249824524, 1.255885362625122, -0.6357324719429016, -1.1724092960357666, -1.0186424255371094, 0.7084417343139648, 0.6920754909515381, -0.5328508615493774, 0.2634749710559845, -1.8801378011703491, -0.5550174713134766, -1.1530364751815796, 2.285027503967285, 0.39547011256217957, -0.902385413646698, 0.27724146842956543, 1.8078627586364746, 1.4644865989685059, 0.536008358001709, -0.8875723481178284, 1.4397298097610474, -0.37457895278930664, -0.013244648464024067, 0.2650049030780792, 0.3180268704891205, 0.5716919302940369, 0.91632479429245, -0.36905208230018616, -0.06122753024101257, 1.8718504905700684, 0.7639775276184082, 0.9335998296737671, -1.3578157424926758, 0.010842449963092804, 0.28530439734458923, 0.8468853235244751, -1.0179473161697388, 0.7381433844566345, -0.5510306358337402, 0.08034662157297134, -1.0251967906951904, 0.1764429658651352, 1.4171379804611206, 1.1054822206497192, -0.5550140142440796, 0.387714684009552, -1.5836635828018188, 0.8563519716262817, -0.7748492956161499, -0.8343685269355774, -1.9359049797058105, 0.3826028108596802, -0.7407167553901672, -1.4176161289215088, 1.5132410526275635, 1.4776581525802612, -1.0769007205963135, -1.0561556816101074, 0.8290079236030579, -0.421354740858078, 0.14553308486938477, -0.5809057354927063, 0.2359107881784439, 0.8020257353782654, 0.6827230453491211, -0.580854058265686, 1.8439552783966064, 0.7391140460968018, 0.4260130524635315, 0.5576738715171814, 1.1300232410430908, 0.1422436684370041, 0.8726825714111328, 1.7817163467407227, 1.0821539163589478, -0.4405755400657654, -2.273484230041504, -0.2822958827018738, -1.3415565490722656, -0.10612349212169647, 1.2133402824401855, 0.9051597714424133, 0.5765457153320312, -0.4531196653842926, -0.5645845532417297, -0.3612937927246094, 1.52743661403656, -1.5623719692230225, -0.4303446412086487, -0.06996254622936249, -0.9632595181465149, -1.3163739442825317, 0.2620891332626343, -1.0084184408187866, 0.055783066898584366, -1.6868107318878174, -1.33783757686615, 0.5568218231201172, -2.9210855960845947, 0.5699360370635986, 0.9637805819511414, 0.5411899089813232, -1.1107264757156372, -0.15543536841869354, -0.8750337958335876, 0.8008649349212646, -0.8868259787559509, -0.4056464731693268, 0.3006642162799835, -1.2448298931121826, 0.19909629225730896, -0.7050583958625793, 0.20161592960357666, 0.621602475643158, 0.2540973722934723, 1.1453015804290771, 0.716188371181488, 1.103074312210083, 1.1087032556533813, -0.5707179307937622, -1.6801520586013794, -1.1415514945983887, -1.2838964462280273, 1.2544491291046143, -0.11752792447805405, 2.7392868995666504, -0.22101572155952454, -0.7531847357749939, 2.55332350730896, 1.3451110124588013, -0.7686513662338257, -0.3481254279613495, -0.8105280995368958, 0.6992071866989136, 0.6708146333694458, -0.3918423354625702, -1.0304763317108154, 0.2862604260444641, 0.2918469309806824, 0.8091040849685669, 1.6002328395843506, -0.43516045808792114, 1.713296890258789, 0.2600979804992676, 0.43656033277511597, 0.6074455976486206, 0.05383126810193062, 0.2591332793235779, -0.47207358479499817, -0.6419841051101685, 0.8987821340560913, 0.1324630081653595, -2.5060782432556152, -1.1714683771133423, 2.362818956375122, -0.09297999739646912, 0.8613824844360352, -0.32565048336982727, -0.19555185735225677, 0.36898502707481384, 0.021145379170775414, -0.9830929040908813, 0.053271736949682236, 1.7700828313827515, 0.518080472946167, 0.11941646784543991, 0.12798729538917542, -0.3473130762577057, -1.29804265499115, -0.8309456706047058, 2.427783250808716, -0.33975493907928467, 0.13581585884094238, 1.5079444646835327, -1.1872100830078125, 2.366266965866089, -1.2252792119979858, 0.7537680268287659, 0.08660020679235458, 0.31385624408721924, 0.11136151850223541, 0.3429093658924103, 0.3213716745376587, 0.7259055972099304, -0.4026462435722351, 1.4952222108840942, -0.7092048525810242, -0.27569639682769775, 0.5827297568321228, -0.5915483832359314, 0.11495345085859299, 0.4161083698272705, 2.1439638137817383, -0.1020728200674057, 0.38944917917251587, 0.08370234072208405, 1.6563435792922974, -1.2874268293380737, -2.3906514644622803, 0.8225623369216919, -2.13493013381958, 1.0526179075241089, -1.5268598794937134, 0.06474380940198898, -0.5659893751144409, 0.053644031286239624, 0.7633297443389893, -0.99081951379776, 0.6662982702255249, -0.850659966468811, -0.4667220711708069, 1.4680198431015015, 0.2567385733127594, 0.4214353561401367, -0.6767321228981018, -0.47687649726867676, 1.1231427192687988, 0.6173872351646423, 1.2336536645889282, -0.1368468701839447, -1.2852648496627808, -0.13773323595523834, -0.276745080947876, -0.3195127844810486, 0.22184516489505768, 0.96235591173172, 0.40502995252609253, -0.09055769443511963, -0.37532129883766174, 0.6201406717300415, 0.580103874206543, 0.9052119255065918, 2.175363779067993, 1.262704849243164, 0.11117568612098694, -0.27486926317214966, -0.512663722038269, -1.426308274269104, 0.023416832089424133, -0.13985055685043335, -2.841489553451538, -1.0332989692687988, -0.9740382432937622, 0.09018220007419586, -1.3856370449066162, 0.9404510855674744, 0.5053255558013916, -1.4700121879577637, -0.28594323992729187, -0.9447095394134521, -1.3004920482635498, 0.3325728476047516, 0.5245766043663025, -0.316137433052063, -0.8441281318664551, 0.14386212825775146, -0.09956538677215576, -0.10056047886610031, 0.04690001532435417, -0.8369056582450867, 1.6449371576309204, 0.3032389283180237, -0.1185360699892044, 2.111337184906006, -0.8246791362762451, 0.46825170516967773, -0.13024337589740753, 1.9531444311141968, -1.7424018383026123, -0.543030321598053, 0.7332839369773865, -1.4599310159683228, -2.1881935596466064, -0.2420523315668106, 0.9649250507354736, -0.6948583126068115, -0.032537974417209625, 0.08870711922645569, 0.8502588272094727, 0.12967103719711304, 0.42163434624671936, 0.5933873653411865, -0.51747727394104, 0.4457462728023529, 0.9977660775184631, 0.16280098259449005, -0.0006481367163360119, 0.3202677071094513, 1.4301095008850098, 1.5497149229049683, 2.2097697257995605, -0.8251486420631409, 1.935990571975708, -0.003856522962450981, 1.0450128316879272, -2.491600751876831, 0.41460439562797546, 1.1806552410125732, -0.2149457335472107, 0.17707625031471252, -0.6419347524642944, -0.6226672530174255, 0.19258734583854675, -0.3662463128566742, 0.7459279298782349, 0.25204330682754517, -0.992937445640564, 0.7022901177406311, -1.6237748861312866, 0.28638797998428345, 0.5942251086235046, 1.458999752998352, -2.1651790142059326, -0.03810357302427292, 0.6970401406288147, -0.31204846501350403, -0.7363483905792236, 0.10054167360067368, 0.5399186015129089, -2.2100632190704346, 1.6693174839019775, -0.863573431968689, -2.051715850830078, -1.9046789407730103, -1.7680954933166504, -1.2123948335647583, -0.29372239112854004, 0.3780366778373718, 0.019626302644610405, 2.0208866596221924, -0.14563393592834473, -0.5728681087493896, 2.033231735229492, -1.2464205026626587, -1.384049654006958, 0.4312807321548462, 1.1196986436843872, 0.8294715881347656, 0.6003395318984985, -1.1242884397506714, -0.7516037225723267, 0.7704324722290039, -2.2063121795654297, -0.03572807088494301, -0.2350027710199356, -1.6362248659133911, -0.05300014838576317, 1.333738088607788, -0.1528060883283615, 0.9151085019111633, -0.5352981686592102, 0.11305274814367294, -1.0260734558105469, -1.0421154499053955, -0.748626708984375, 0.5340220928192139, -0.14662203192710876, 0.8045331239700317, 0.1302882730960846, 0.7771106362342834, 0.3583034574985504, 1.835365653038025, -0.628725528717041, 0.23313167691230774, -0.8366831541061401, 0.46877071261405945, -1.026052474975586, -0.5040150284767151, -0.23198576271533966, -0.5917167663574219, 0.7933621406555176, -1.1042839288711548, 0.9016356468200684, -0.7726461291313171, -0.5608633756637573, 0.6015287637710571, 1.0135090351104736, 1.6024901866912842, 0.7117050290107727, 0.18777833878993988, -0.6721252202987671, -0.46269312500953674, -0.3270012140274048, -0.366694837808609, -0.8966585993766785, 0.8567696213722229, 0.764649510383606, 0.517936110496521, 0.9317666292190552, -1.0022016763687134, -0.3589188754558563, 0.48985755443573, 0.6738247871398926, 0.05446615070104599, -1.179094672203064, 1.0316224098205566, 0.355415016412735, 0.30769655108451843, 1.1930334568023682, -0.17901673913002014, 0.8637163639068604, -0.6319319605827332, 1.2336078882217407, 1.946144938468933, 0.4966574013233185, 0.3566538393497467, -0.5394519567489624, 0.3424241840839386, 1.3976539373397827, 0.5224899053573608, 0.6499911546707153, -0.22447673976421356, 1.2772557735443115, -0.3109838366508484, -0.8110602498054504, 0.9930561184883118, -1.0222280025482178, 0.3280036747455597, 0.09188789874315262, 1.3074232339859009, -0.6563411951065063, -1.684631109237671, 0.9940298199653625, -0.07175718247890472, -0.17763160169124603, 0.8270723223686218, 0.3392047882080078, -0.16180333495140076, 0.6913135647773743, 1.783018708229065, 0.20300327241420746, -0.06512550264596939, 0.9478527903556824, -0.3567957282066345, -0.4742738604545593, 1.3522597551345825, -0.19176647067070007, 0.4430682957172394, 0.2973378002643585, 1.8148647546768188, 0.8443459272384644, -1.406469464302063, 0.547126054763794, -0.07827023416757584, 2.6519012451171875, 0.32017719745635986, 1.708522081375122, -0.825421929359436, -0.6321728229522705, 0.5253668427467346, 0.013365247286856174, 2.222625255584717, 0.8975409269332886, -0.6185562014579773, -0.7135891914367676, -0.7191928029060364, 1.188378095626831, 1.4590359926223755, -0.31751957535743713, -1.0358402729034424, 0.8801560997962952, -1.528599739074707, 0.5439679622650146, 0.04613683000206947, -1.16487455368042, -0.9010559320449829, 0.8344169855117798, -2.3052337169647217, 1.023274302482605, 1.4792141914367676, -0.22795970737934113, -0.40090271830558777, -0.4836651384830475, 1.047115683555603, -1.4942842721939087, 1.2801138162612915, -1.3440961837768555, 0.2336936891078949, -0.44126901030540466, -1.3098281621932983, 2.0900075435638428, 0.48746055364608765, 0.0474608913064003, -0.13748139142990112, 0.21025004982948303, -0.5744488835334778, 0.8095516562461853, 1.792710542678833, -0.2347731739282608, -0.14339511096477509, 1.749079704284668, -1.0553394556045532, 0.3594784140586853, -0.6899780631065369, 0.490466445684433, -1.3259183168411255, 1.7361677885055542, -0.28365951776504517, 1.8719183206558228, 1.2666246891021729, -1.5941660404205322, -1.054659366607666, -0.4589684009552002, 0.024130508303642273, 0.49450695514678955, -0.2230251133441925, -0.6810110211372375, -0.44695594906806946, -0.5750892758369446, 1.7564152479171753, 2.3758718967437744, 0.767657458782196, -0.3239046335220337, -0.7645420432090759, 0.34411558508872986, 0.7779232859611511, 0.6542329788208008, 0.5712509155273438, -0.6117138862609863, 0.35496973991394043, -2.223200559616089, -0.6714848279953003, 0.22838376462459564, -1.3868669271469116, 0.8975765109062195, 0.7024235725402832, -0.18105128407478333, 0.47446873784065247, 0.6239727735519409, -0.9539133310317993, -0.01543855108320713, 0.7156455516815186, -0.02332998253405094, 1.693499207496643, 0.8917710781097412, -0.7215944528579712, -0.715070366859436, 0.038789037615060806, -1.8536814451217651, -1.7319401502609253, 0.8330410718917847, -0.24994631111621857, -0.6842969059944153, -1.2958669662475586, -1.857412576675415, -0.08600020408630371, 0.871468722820282, 0.6466059684753418, -0.2197718322277069, 1.4469044208526611, 0.5131757259368896, -0.48136234283447266, 0.49324068427085876, -0.7912005186080933, 2.1515862941741943, -0.1337951272726059, -2.5523645877838135, 0.837891697883606, 0.06701337546110153, 0.2050638645887375, -1.1503944396972656, -1.0704995393753052, 0.5461788773536682, -1.4015955924987793, 0.02730114385485649, 0.580621600151062, 0.28276413679122925, 0.24307945370674133, 0.08890961110591888, -0.8678317070007324, 3.294654130935669, 0.20556896924972534, 1.1787910461425781, 0.7100933790206909, -0.3753175735473633, -0.7348731756210327, -1.8954728841781616, 0.6345682740211487, 0.18936672806739807, 2.1591110229492188, -0.378648042678833, -1.331201195716858, -1.3470275402069092, -0.18349842727184296, 0.36244937777519226, 0.06963885575532913, -0.5426696538925171, 0.007776347454637289, -0.6550347805023193, 1.859891653060913, -0.6999890208244324, -0.00834033451974392, -0.5755024552345276, -0.7275328636169434, -0.1378234177827835, -0.6938298940658569, 0.7247447371482849, 0.3421024680137634, 0.4600840210914612, -0.3702310621738434, -0.24036815762519836, 0.20600424706935883, 0.3222619593143463, 1.4053325653076172, -0.6194406747817993, -0.3844054043292999, -0.23282918334007263, -0.35128504037857056, -1.4081426858901978, -0.4834989607334137, -0.3756949007511139, -1.0604872703552246, -0.21887600421905518, 0.9287957549095154, 0.0750465989112854, -0.8629288673400879, 1.9044601917266846, -1.570164680480957, 0.3100382089614868, -1.2471574544906616, 0.15154723823070526, 0.04282408207654953, -0.7855229377746582, 0.29420480132102966, 0.5803459286689758, -0.5963774919509888, -0.04127596318721771, 1.553666114807129, -1.4247157573699951, -0.8475951552391052, -1.9142472743988037, -0.12560850381851196, -0.7045861482620239, 1.2991458177566528, 2.177478790283203, -1.7569276094436646, 0.5160446166992188, 1.9231650829315186, -0.3166393041610718, -1.678608775138855, -0.2888888418674469, -1.495193362236023, 1.2909328937530518, -0.4669557213783264, 0.5543285012245178, -0.36593568325042725, 0.3401498794555664, -0.3727957606315613, -1.5699659585952759, 0.008974439464509487, -0.7121371030807495, 0.324088454246521, 0.33122387528419495, -1.5741615295410156, -0.026219015941023827, -0.651168704032898, -0.17980481684207916, 0.8128421902656555, -0.11828788369894028, -0.22877424955368042, -0.18928547203540802, -0.5223848223686218, -0.5394722819328308, 0.5045123100280762, -0.7376238703727722, 0.38549256324768066, 0.2478019744157791, -0.7187842726707458, 0.44601497054100037, 0.8694702386856079, -0.14609253406524658, 0.12543153762817383, 0.379538357257843, 0.3897460997104645, 0.9887920618057251, -0.10248415917158127, 0.6754753589630127, 0.09083119034767151, 1.1337530612945557, 0.020862853154540062, -0.7401097416877747, 0.26639118790626526, 0.15400907397270203, 0.7941344380378723, -0.9558496475219727, -0.0029631115030497313, 0.45777076482772827, 0.045060206204652786, 0.09486692398786545, 0.16769206523895264, 1.4473075866699219, 1.0779324769973755, 1.028784990310669, 1.0209815502166748, -1.0011179447174072, -1.5147212743759155, -0.9278839230537415, -1.3810921907424927, -0.21265798807144165, -0.4552610218524933, -2.3889620304107666, 2.555729627609253, 0.33818286657333374, 0.5600782632827759, -0.8286975622177124, 2.0597946643829346, -0.20675627887248993, 1.1141364574432373, -0.9133051037788391, -0.23766237497329712, -0.5459203720092773, -1.5779402256011963, 0.27171754837036133, 1.7653881311416626, -1.2589359283447266, -0.23467308282852173, -0.3017869293689728, 0.7840226888656616, -0.3044538199901581, -0.14938578009605408, 1.1207103729248047, 0.15231890976428986, -0.6704217791557312, -1.7834043502807617, 0.43955230712890625, -0.6594783663749695, -2.381925344467163, -0.1330142468214035, -0.30718135833740234, 0.5821070671081543, -0.22574612498283386, -0.7530494928359985, -0.10090050846338272, -1.8137269020080566, -1.3006891012191772, -1.1503864526748657, 1.5920636653900146, 0.6546672582626343, 0.12356148660182953, 0.25538209080696106, 0.33541831374168396, -0.659208357334137, 0.3184414207935333, 0.5796240568161011, 0.5131707191467285, 0.19365356862545013, -0.5496184229850769, 0.6661006212234497, -0.731338381767273, -0.8288446068763733, -0.19349436461925507, -0.5944285988807678, 1.8992860317230225, -0.5472533106803894, -0.5467808246612549, 0.7774462103843689, 0.9316986799240112, 1.3355417251586914, 0.4444003403186798, 1.6450355052947998, 0.9827561974525452, -1.3714954853057861, -0.962714672088623, -1.1944516897201538, -0.020081905648112297, -0.3124006688594818, 0.27238377928733826, -0.04783957824110985, 1.1481971740722656, -1.8873217105865479, 0.42715731263160706, -0.10918362438678741, 0.6340366005897522, 2.488032102584839, 1.4281588792800903, 0.2072458118200302, 0.021109532564878464, -0.5223160982131958, 1.2046387195587158, 1.3903636932373047, -1.1217842102050781, 0.12221863865852356, -0.7514878511428833, -1.8671791553497314, 2.1531431674957275, 0.788360595703125, 1.5229846239089966, -0.13081784546375275, 1.4215980768203735, -0.487638920545578, -0.37547025084495544, -1.6295742988586426, 0.6095405220985413, 0.7063122391700745, -0.7494983673095703, -1.8733935356140137, -0.9643409252166748, -0.2681070566177368, 0.2887987494468689, 1.0377426147460938, 0.6361512541770935, 0.4239782392978668, -0.4564361870288849, 0.9156156778335571, 0.5203801989555359, 0.9376048445701599, 0.7037105560302734, 0.4230318069458008, 0.9041181206703186, 1.6877638101577759, 0.5113264918327332, -0.6219601035118103, -1.858282208442688, 0.04330715909600258, -0.18924281001091003, -0.43990013003349304, 1.075577974319458, -0.4522814154624939, -0.4294598698616028, 0.20563748478889465, -0.8968658447265625, 0.32127851247787476, -0.42827099561691284, 0.7806114554405212, -1.0239145755767822, 1.0694929361343384, -0.467519074678421, -0.6894469857215881, -0.06339428573846817, -0.268335223197937, 1.8605214357376099, -0.9208871722221375, 0.08283717930316925, 2.138050079345703, -0.3781334459781647, 0.07313220202922821, -2.131460666656494, -1.0766452550888062, 0.24002927541732788, 0.9335526823997498, -1.8409838676452637, -0.06638751178979874, -0.23331035673618317, -1.4837671518325806, -0.11273414641618729, -1.5502054691314697, 0.9384424090385437, -1.0092244148254395, 0.883114218711853, -0.02279820665717125, 0.47239139676094055, 1.4182454347610474, 0.3159138262271881, -1.3537349700927734, -1.091693639755249, -1.4554710388183594, 0.16462795436382294, 0.723151445388794, 0.7857590317726135, -1.6997085809707642, -1.572357177734375, -0.38923129439353943, -0.6453201174736023, -0.26612523198127747, 0.6358294486999512, 0.4573134183883667, -1.5050408840179443, -1.2942602634429932, 0.22565264999866486, -0.4989197552204132, -0.4126327335834503, -0.8513930439949036, -0.39317378401756287, -2.5233309268951416, 0.7354910969734192, 0.10612510144710541, -0.012436852790415287, 0.3166567385196686, 1.4110735654830933, 0.35380786657333374, 0.21854303777217865, -1.8393785953521729, 0.6149824857711792, -0.28163039684295654, 0.5173450112342834, -0.6462234854698181, -0.542102575302124, 0.06360668689012527, -0.07214315980672836, -0.5580050945281982, 0.12500934302806854, -0.6125381588935852, -1.246084213256836, 0.9275599122047424, 0.06740855425596237, 1.3237175941467285, -0.37248826026916504, -0.5202909708023071, 1.0047650337219238, 2.621994972229004, -0.2367638200521469, -1.4777106046676636, -0.2934321463108063, -1.9652949571609497, 0.8338398337364197, -0.8010143041610718, -0.9234262704849243, -0.501943826675415, -2.3636372089385986, -0.2527976334095001, 1.2468886375427246, 0.5897171497344971, 0.7928855419158936, 0.06500763446092606, -0.3441983759403229, 1.4641867876052856, -0.36053401231765747, -1.2849892377853394, 0.033563338220119476, 0.09502074867486954, -0.8654170036315918, -0.9308885335922241, 0.5061210989952087, 0.0660192221403122, -0.7944002747535706, -1.2254217863082886, 0.7592709064483643, -0.10992362350225449, -0.20386210083961487, -0.13282510638237, -1.4057115316390991, -0.9308193325996399, -0.6775685548782349, 2.443598985671997, -0.8410555720329285, -1.905374526977539, 0.4575849771499634, -0.4799782335758209, -0.06229065731167793, 0.8528664112091064, 0.3984197974205017, -0.30840906500816345, -0.9019386768341064, 0.9105609059333801, -1.32101309299469, 0.979309618473053, -0.6425616145133972, 0.5167280435562134, 0.5692484974861145, -1.1503132581710815, 1.07071053981781, 0.5013393759727478, 0.9240654706954956, 1.5751583576202393, -0.35604050755500793, 0.12776778638362885, -2.878340244293213, 0.8453310132026672, 0.409475177526474, -0.9583192467689514, 1.1291719675064087, 0.5248925089836121, -1.4553964138031006, 0.47613757848739624, -1.4606413841247559, 0.3508917987346649, -0.8799891471862793, -0.767412006855011, -0.15020228922367096, -0.8993757367134094, 1.3683786392211914, -0.35594460368156433, -0.03546528145670891, 1.694891333580017, 0.7827444076538086, 0.49306538701057434, 0.37146881222724915, 0.7861324548721313, 1.1787068843841553, -0.2707711458206177, 0.9017754197120667, -0.31805017590522766, -0.689935564994812, 0.46381890773773193, -0.4211612343788147, -0.7192704677581787, 1.4511759281158447, 0.9157443046569824, 0.07943155616521835, -0.4991462826728821, -0.14135253429412842, -1.0417046546936035, -0.5163418054580688, -0.8743235468864441, 2.142744541168213, -0.9630798101425171, -0.6494487524032593, 0.9801667332649231, -0.6624339818954468, 0.44390738010406494, -0.6110116839408875, -1.2646492719650269, 1.4881573915481567, -0.9795570373535156, -0.31288665533065796, -0.07512841373682022, -0.23670551180839539, 2.4838337898254395, 1.2275221347808838, -0.6086598634719849, 0.3442148268222809, 1.7175956964492798, -1.2171881198883057, -0.5289185643196106, 0.9081944227218628, -0.6741026639938354, 0.4654349386692047, -2.188129425048828, -0.0819425880908966, -2.962515354156494, 1.3605263233184814, 1.5981794595718384, 0.7700550556182861, 0.7783684730529785, 1.9962977170944214, 0.5716288685798645, -0.5967018008232117, 0.6409386992454529, 1.223840355873108, 0.5730348229408264, -0.2428368330001831, -1.9839015007019043, 1.523672103881836, -0.6293498277664185, 1.1520793437957764, -0.4197787046432495, -1.695075273513794, -0.802056610584259, -0.7980734705924988, -0.6185319423675537, -0.6361202597618103, 1.5841213464736938, -0.3786020576953888, 0.1889045536518097, 0.7975828647613525, 0.1958606243133545, 1.0716947317123413, -1.4354679584503174, -1.857615351676941, -0.06952988356351852, 0.1136309951543808, -0.1574603170156479, 0.9327269792556763, 0.07263967394828796, 0.4777020812034607, -1.903496503829956, 1.4677611589431763, 0.5339165925979614, -0.38944628834724426, -0.899607241153717, -2.1717422008514404, 0.097689189016819, 0.2183047980070114, -0.49352937936782837, 0.09044332802295685, 0.6401672959327698, -1.2224425077438354, 0.031098922714591026, -0.4616575837135315, -0.7920975089073181, 0.8514627814292908, -0.34610798954963684, -0.4166150391101837, 1.1196479797363281, -0.581558883190155, 1.3530699014663696, 0.880472719669342, -0.6069532632827759, -1.2459977865219116, -1.0553019046783447, 0.327273964881897, -1.0094883441925049, 0.5186235308647156, 0.23317140340805054, 1.5017026662826538, 1.7582857608795166, 0.495683491230011, -1.0025895833969116, -0.4339440166950226, 1.946307897567749, 0.6122338771820068, 0.4199950098991394, -0.24672432243824005, 0.09476106613874435, -0.08552665263414383, -1.9805264472961426, -0.7491754293441772, -1.4605913162231445, 1.0571154356002808, -0.6355982422828674, -0.22831463813781738, -0.8719369173049927, -0.8935092091560364, -0.012107647955417633, -0.6389811635017395, -0.8037341833114624, -0.5531438589096069, -0.7416973114013672, 0.29945486783981323, 0.7079921364784241, 0.6145376563072205, 0.6140226125717163, 0.31971532106399536, 0.0868721678853035, -1.9020127058029175, 0.6040278077125549, 0.45495814085006714, -0.995299220085144, -0.7861272096633911, -0.6452836990356445, -1.4776027202606201, 0.15239454805850983, -0.31318408250808716, 1.0308595895767212, 0.8360450267791748, -0.8000497817993164, -0.9197264313697815, 0.22879935801029205, 0.012428448535501957, 0.3198237419128418, 1.7728945016860962, -1.6596335172653198, -0.05749610811471939, 1.3918392658233643, -0.14814803004264832, -0.22969050705432892, 0.3416227102279663, -0.9324293732643127, 0.5997862219810486, 0.9185950756072998, -0.016580048948526382, 0.6079498529434204, -0.16717447340488434, 0.3359678387641907, 2.0239169597625732, 0.5079289078712463, -0.016033915802836418, 0.16293448209762573, -0.7654977440834045, 0.760523796081543, 0.9955999255180359, -1.1669918298721313, 0.9155970215797424, 1.4389100074768066, 1.005350947380066, -0.2207481414079666, 0.5318811535835266, 1.805808424949646, 1.6429930925369263, -0.3636464476585388, -1.3457512855529785, 1.0338784456253052, -0.3539689779281616, -0.5449389815330505, 0.044129952788352966, 1.86131751537323, -1.1527849435806274, 1.60882568359375, -0.2227247655391693, -0.7733178734779358, -0.8430190086364746, -0.4925467371940613, 1.0139226913452148, 0.007566208951175213, -0.2993585765361786, -0.818458616733551, 0.9311596155166626, 0.6015050411224365, 0.013186472468078136, 0.6288920044898987, -0.38856086134910583, -2.208683729171753, 1.0122075080871582, -2.0131542682647705, 0.324541300535202, -1.0411995649337769, 1.1023627519607544, 0.8638399243354797, 0.4084259569644928, -0.6893675923347473, 0.6558060050010681, 2.1342356204986572, -0.283476859331131, 1.1798299551010132, 0.7121285796165466, 0.5064667463302612, -0.8560323119163513, 0.8604711890220642, -0.5882757902145386, 0.6952002048492432, -0.47419050335884094, 0.39320769906044006, -0.9377445578575134, -0.7234103679656982, 1.4883521795272827, -0.3673745095729828, -0.5546819567680359, 0.8324361443519592, 0.2726335823535919, 0.5933194756507874, 0.6188690662384033, 0.023751698434352875, -0.13528411090373993, -0.24989458918571472, 1.539465308189392, -0.2423056960105896, 0.2785915732383728, 2.5563313961029053, -0.42408421635627747, -0.3832038938999176, -0.40519964694976807, 1.5413293838500977, -2.0923893451690674, 1.5378081798553467, 0.7410598993301392, 1.0085890293121338, -0.8370955586433411, -0.13079392910003662, -0.0009884415194392204, -0.8312522768974304, 0.26750683784484863, 0.3483791649341583, 0.1296205371618271, -0.9610861539840698, -0.04060923680663109, -1.412360668182373, 1.4785232543945312, -0.2795713245868683, -0.42126667499542236, 0.098933644592762, 0.9757593870162964, -0.8343865275382996, -0.42956051230430603, 1.618237853050232, -1.3265453577041626, -0.7630175352096558, 0.12532305717468262, -1.032538652420044, 1.3911298513412476, 1.0047568082809448, -0.29741042852401733, 0.42523565888404846, 0.27885863184928894, 1.7788654565811157, 1.7632794380187988, -1.37309730052948, 0.018503619357943535, -1.1130917072296143, -0.7194503545761108, 0.28023791313171387, -0.035926979035139084, 0.3276905417442322, 0.6038995981216431, 2.9164342880249023, 0.38904017210006714, 0.06458567082881927, -0.5499049425125122, -1.3868536949157715, 1.3107353448867798, -0.6300493478775024, 0.6615059971809387, 0.37139132618904114, 0.37323740124702454, 1.5243146419525146, 0.5693938136100769, 0.5812340378761292, -1.1143187284469604, -0.4253849983215332, -0.39564427733421326, -0.49458417296409607, -1.5009214878082275, -1.5166610479354858, 0.3740864396095276, -1.1689224243164062, -1.8211933374404907, 0.2858569025993347, 1.556215524673462, 1.1036291122436523, -1.9113175868988037, 0.40178313851356506, -1.827028751373291, -0.742132842540741, -0.07431967556476593, 0.3232354521751404, -0.9214451313018799, 0.46885645389556885, 0.9525285959243774, 1.0341229438781738, 1.348186731338501, 0.07624029368162155, -1.0823460817337036, 0.4665206968784332, -1.2445937395095825, 0.9188886284828186, 1.5272091627120972, -0.4805237352848053, 0.5582857728004456, 0.7726848721504211, 2.2080488204956055, -0.8542404770851135, 0.3182491660118103, -0.5278177857398987, 0.0521436482667923, 0.49459901452064514, -1.0135785341262817, -0.5869866609573364, -0.853840172290802, 0.10173994302749634, -2.505237579345703, -0.9634736776351929, -0.14203020930290222, 1.065585970878601, -0.6786116361618042, -0.9132910966873169, -0.5741210579872131, -0.4021880328655243, -0.22602424025535583, 1.1041107177734375, 0.3160936236381531, 0.3128325343132019, 0.9684688448905945, 0.04908614605665207, 0.34665945172309875, -0.2852781414985657, 0.0014721248298883438, 0.7373443245887756, 0.4301626682281494, 2.1540210247039795, 0.29066720604896545, -2.063908338546753, 0.7646302580833435, 1.2965631484985352, -0.48242586851119995, 2.757824420928955, 0.005548837129026651, 0.9197604060173035, 0.2287740558385849, -1.5179541110992432, 0.4264994263648987, -0.2516898810863495, -0.15425975620746613, -0.7635429501533508, -0.24403677880764008, 0.9567351341247559, 0.6401928067207336, 0.403645396232605, 0.02868579514324665, 1.8114277124404907, -0.8378227353096008, 0.757564127445221, 0.2913810908794403, -2.3168604373931885, 0.3895286023616791, 0.3416441082954407, -0.3401678800582886, -0.3587896227836609, -1.0893546342849731, 0.44623076915740967, -1.6342204809188843, -0.7072283029556274, 1.3700170516967773, 0.7684838771820068, -0.6907606720924377, 0.3334708511829376, 0.6897982358932495, 0.7125222086906433, -0.5529766082763672, 1.1676146984100342, -1.7856183052062988, -1.0636658668518066, -0.27710434794425964, -0.2050405740737915, 0.4741441011428833, 1.1293883323669434, 0.9428966641426086, -0.04884680360555649, -0.802485466003418, 0.8315169811248779, -0.26525354385375977, -0.5016194581985474, -2.5687429904937744, -1.1580837965011597, 0.6531504392623901, 0.46877244114875793, 1.0805102586746216, 1.0072218179702759, 0.5697975754737854, 0.4388163089752197, 0.6247513890266418, -0.1311575025320053, -0.40047305822372437, 1.377497911453247, 1.6193979978561401, 0.27367091178894043, 0.39187565445899963, -1.5298110246658325, 0.841773271560669, 1.3537641763687134, -0.2122945934534073, 0.01836400479078293, -1.6225988864898682, 0.28690165281295776, -0.6189984679222107, -0.6403136849403381, 0.3998417854309082, 0.7178370952606201, 1.6244406700134277, 1.3166730403900146, -0.6241859197616577, -1.5714446306228638, -0.12096177041530609, 0.832998514175415, -1.9392762184143066, 1.2749141454696655, 0.07191499322652817, -1.6720162630081177, 0.7335391044616699, 0.07526258379220963, -0.4940871596336365, 0.9422717094421387, 0.5914108157157898, 0.8728535771369934, -2.1250436305999756, 0.44079166650772095, -0.194401815533638, 0.17408378422260284, 1.26467764377594, 0.22831009328365326, -0.49975377321243286, -1.3375838994979858, 0.6154142618179321, -1.71812903881073, -1.7354621887207031, -0.6474879384040833, 0.2058674693107605, 0.15664221346378326, -0.2299456000328064, 0.5967036485671997, -1.1165838241577148, -0.6564787030220032, 1.4789530038833618, -1.9151109457015991, -0.7703482508659363, 0.6928050518035889, 0.7928754687309265, 0.11743953078985214, -3.261484146118164, -0.3684844970703125, 0.45002686977386475, -0.7920148968696594, 0.2916620969772339, 1.660104513168335, -0.47613033652305603, -0.6943461298942566, 0.3091633915901184, -1.1837382316589355, 0.6374203562736511, -0.03378533571958542, 0.41989463567733765, -0.7863481640815735, 1.1764642000198364, -0.7005183696746826, 0.16987843811511993, 0.07568852603435516, -0.8935254812240601, -1.3146013021469116, 1.1944596767425537, -0.013967781327664852, 1.7961807250976562, 0.30868199467658997, 0.4923837184906006, 0.19480237364768982, 0.33911868929862976, -0.8605494499206543, 1.3395569324493408, 1.1991605758666992, -0.05763569101691246, 1.1696914434432983, -0.6451939344406128, -0.3447769284248352, 0.49218833446502686, -0.3868449926376343, 0.2284378707408905, 0.3519917130470276, 0.09439067542552948, 0.5144944787025452, -0.3606813848018646, -0.38858258724212646, 0.319042444229126, -1.2733356952667236, 0.6838832497596741, 0.9773091077804565, -0.524844229221344, 2.20204496383667, 1.4447604417800903, 1.1725788116455078, 0.7306762933731079, -1.8701274394989014, 0.015349327586591244, -1.2675138711929321, -0.5372722744941711, 0.24698325991630554, 0.6426059603691101, 0.712444007396698, 0.011091346852481365, -0.6673981547355652, -1.1332758665084839, -0.4668477475643158, 1.401516079902649, 0.5337003469467163, -0.9703924655914307, -0.9179362654685974, -1.7663182020187378, -2.093848466873169, -0.013826608657836914, -1.4952471256256104, -1.24904203414917, -2.2043163776397705, 0.362712562084198, 0.20616519451141357, 1.0377452373504639, 0.2530311346054077, -0.42210251092910767, -0.4225311577320099, 1.224326491355896, -0.302355021238327, 1.3200584650039673, 1.460472583770752, -0.35953471064567566, 0.10395537316799164, 0.19090066850185394, 0.4757760167121887, -2.22552227973938, 0.6566919684410095, 0.06266144663095474, -1.0768111944198608, 0.8266634345054626, 0.04707137122750282, -0.3567282259464264, -1.4502716064453125, 0.21096502244472504, -0.7205178737640381, 0.4585849344730377, 1.4979482889175415, 0.9975311160087585, -0.366542249917984, 0.7281413078308105, -0.21801307797431946, -0.3902105987071991, -1.3390512466430664, 0.7370126247406006, -0.5321166515350342, -0.22792111337184906, 0.23063252866268158, -0.3472278118133545, -0.108125239610672, -0.5333174467086792, -0.7294542193412781, 0.7645984888076782, 0.1715739667415619, 0.8413034081459045, 0.1799096018075943, -2.210268974304199, -0.053457111120224, 0.1167340874671936, -0.34164488315582275, -2.2650909423828125, 0.49882152676582336, 0.27686551213264465, -0.487150102853775, 1.5119718313217163, -0.862313985824585, 1.3377668857574463, -1.003105640411377, -2.167278528213501, -2.1021738052368164, 0.8234907984733582, 1.2133034467697144, 0.911416232585907, 0.6467170119285583, -0.37155357003211975, 0.007141012232750654, -0.3163916766643524, 1.6009801626205444, -0.5755441784858704, 1.7741758823394775, 0.5706139206886292, -0.4108321964740753, -0.9232195019721985, 1.772351861000061, -1.6986870765686035, 1.233012080192566, 0.6266847848892212, 0.6866705417633057, -1.295501470565796, -0.7449496984481812, -2.0967886447906494, 0.18064355850219727, 0.4421190321445465, 0.4558955132961273, 0.6136518120765686, -0.35658353567123413, 2.2244362831115723, 0.6005038022994995, 2.3379108905792236, 1.4123600721359253, 0.6183861494064331, 0.6551182866096497, -0.681035578250885, 0.7312221527099609, -0.7382769584655762, 1.3360888957977295, 0.045935213565826416, -2.3412094116210938, -1.4177210330963135, 0.29847779870033264, -0.5045660734176636, 1.1284470558166504, -0.509022057056427, -0.1780213564634323, 1.627266526222229, -0.3941112160682678, 0.41455739736557007, 0.5304538607597351, 0.06914041191339493, -0.7459494471549988, -1.5524259805679321, -1.0001611709594727, -0.21912801265716553, 0.5467725396156311, 0.5006713271141052, 0.38324597477912903, 0.630679726600647, -1.4961392879486084, 0.11089956015348434, -2.595026969909668, -0.5336017608642578, 1.4304590225219727, -0.2495758831501007, 1.5755671262741089, 1.5403432846069336, 1.7109942436218262, 0.9020156860351562, 0.7550788521766663, -0.5019976496696472, 1.8923195600509644, 1.4075666666030884, 0.5411873459815979, -0.9425357580184937, -0.2688034772872925, -0.08035580068826675, -2.606706380844116, 0.6790170073509216, 1.102283000946045, -0.8149722218513489, 0.21244513988494873, -0.5426032543182373, 0.07261528074741364, -0.5263810157775879, -0.1850576549768448, -1.3438982963562012, -0.5170567035675049, 0.47927284240722656, 1.5763258934020996, 0.5812920928001404, -1.4068284034729004, 0.6569325923919678, -1.6512317657470703, -0.555485188961029, 1.0981886386871338, -0.025378113612532616, -2.1091561317443848, 1.9288514852523804, 1.5103172063827515, -2.287202835083008, -0.07541122287511826, -0.8359375, -0.8114975094795227, 0.45392993092536926, 0.07202769070863724, 1.45307457447052, -0.09708062559366226, -0.44100862741470337, -0.612190306186676, -0.7622089385986328, 0.49808910489082336, 1.2066859006881714, -1.5748968124389648, 0.3661332130432129, 0.5278970003128052, -0.30500009655952454, -0.19181297719478607, -2.0795540809631348, -0.4529778063297272, 0.9289711117744446, 0.39517587423324585, 2.131160259246826, 1.3530933856964111, 0.2600548267364502, -1.068203330039978, -0.2886357307434082, -1.7764439582824707, -1.5850759744644165, 0.8916791081428528, 1.072124719619751, -0.009320053271949291, -0.9117631316184998, -0.23131093382835388, -0.5430448055267334, -0.01471287477761507, 0.9409711956977844, 1.7134568691253662, -0.32101964950561523, -0.2576461434364319, 2.5076396465301514, 0.2153085321187973, 1.9947866201400757, 0.914775013923645, 1.527544379234314, 2.0793838500976562, 0.2543336749076843, 0.31956538558006287, -0.6537377238273621, -0.4862818419933319, 0.13941043615341187, -0.32357990741729736, 0.6839008927345276, -1.721679925918579, 1.0950688123703003, 0.4135105311870575, 0.03215370327234268, -1.7734397649765015, -0.3987395763397217, -0.9313387870788574, 0.34947821497917175, -1.6518118381500244, -0.8472205996513367, -0.2786291241645813, 0.11434252560138702, 1.8516182899475098, 0.7456128001213074, 0.39536115527153015, 0.19508987665176392, 1.1077663898468018, -0.16663619875907898, -0.5040059089660645, 0.3566907048225403, 2.167217254638672, -0.2136273980140686, -1.4744795560836792, -1.1026331186294556, 0.17854920029640198, -1.3580726385116577, 0.7161024808883667, -0.18250109255313873, -1.6414482593536377, 0.6204075813293457, 0.15155544877052307, 0.16475243866443634, 0.1867964118719101, 1.635960340499878, 1.646775484085083, 0.04323122650384903, 0.604182779788971, -1.6181138753890991, -1.6685885190963745, 0.935184121131897, -0.47827741503715515, -0.021771153435111046, -1.0265320539474487, -1.5591986179351807, 2.254791259765625, 1.7410544157028198, 1.343131422996521, -3.3810722827911377, 1.3511625528335571, 0.978321373462677, 1.4537549018859863, -1.5093499422073364, -0.9117449522018433, -0.6927554607391357, 1.138324499130249, 0.7862369418144226, -0.15309688448905945, -1.7946208715438843, -0.5328088402748108, -0.13148225843906403, -0.4415993392467499, 0.11935126781463623, -0.07697618752717972, -0.7076385021209717, -0.9738108515739441, 0.5770806670188904, 0.5938187837600708, 1.272377371788025, -0.43022727966308594, 1.374815583229065, 2.5983524322509766, -0.17771586775779724, -0.7014368176460266, -0.7176139950752258, 0.49278131127357483, -1.1363786458969116, 1.2347978353500366, -0.7744678258895874, -0.03501705452799797, 0.6447508335113525, 0.4026908278465271, 0.3645099699497223, -1.2938027381896973, -0.9372695088386536, -0.8848037123680115, 0.23940901458263397, -0.534202516078949, -0.955815851688385, 0.6448238492012024, -0.3580479323863983, 2.0352022647857666, -0.16257579624652863, 0.07904522866010666, 0.13027623295783997, -0.6080604791641235, -1.9271259307861328, -0.17613936960697174, -2.8627071380615234, 0.08884578198194504, -0.6027976870536804, -0.9967086315155029, -1.726093053817749, 1.506987452507019, -0.12433664500713348, 1.022108554840088, -1.636126160621643, 1.2397098541259766, -0.6230323910713196, -0.26023077964782715, 0.24522753059864044, -1.6639434099197388, -1.4878162145614624, 1.480076551437378, 1.385082483291626, 0.10290027409791946, -0.5145221948623657, -1.5803531408309937, 1.1675841808319092, 0.5721298456192017, 2.4250106811523438, 1.052435278892517, -0.5302598476409912, -0.08781617879867554, 0.5446808934211731, -0.5865575671195984, 0.013325582258403301, 0.26596012711524963, -1.0550957918167114, 2.2926108837127686, 0.22692739963531494, 2.139653205871582, -0.0718834400177002, 1.256468653678894, -0.4112045466899872, 0.7559480667114258, 0.0010484587401151657, 1.1728745698928833, 0.7095919847488403, -1.094696044921875, 0.24833250045776367, -2.8635201454162598, 0.3567048907279968, -0.020630549639463425, -0.7723186612129211, -0.8119666576385498, 0.3913740813732147, 0.4573105573654175, -0.7966392636299133, 0.9918873906135559, -1.0386959314346313, 0.4134139120578766, -2.172321081161499, 0.8301123976707458, -0.7986763715744019, 0.1752656102180481, -1.7438682317733765, -0.5659018754959106, -0.6890517473220825, -1.3534398078918457, 0.3586873710155487, 0.3115476965904236, -1.028173804283142, 1.0443159341812134, -1.3521147966384888, -0.18525287508964539, -0.9690015316009521, 0.18134945631027222, 0.8580228686332703, -0.5890118479728699, 0.827241063117981, -0.4503267705440521, 1.1159898042678833, 0.491031676530838, 0.061987172812223434, 0.9510639905929565, -1.1629042625427246, -0.38859790563583374, 3.196336030960083, -1.2317777872085571, -0.38851475715637207, -0.5524706244468689, 0.7876585721969604, -0.6329213976860046, 1.5454795360565186, 0.4417986571788788, -0.11938595026731491, 0.42328789830207825, 0.03324681147933006, 1.0078930854797363, 1.5215821266174316, 1.9888406991958618, 1.0743192434310913, 0.5448423624038696, -1.1570676565170288, 0.18189361691474915, -0.8053848743438721, -1.295854091644287, -0.12483339011669159, 1.437648892402649, 2.1897733211517334, 0.6155409216880798, 0.7612625956535339, 1.3930834531784058, -0.8009979128837585, -0.2431717962026596, -1.0120868682861328, -0.37059950828552246, 0.1565888375043869, -0.7292688488960266, 0.6602094173431396, 0.47235608100891113, -0.7026609778404236, -0.8422907590866089, -1.778207778930664, -0.17839297652244568, -0.09842655807733536, -0.8370590209960938, 0.6975393891334534, -1.0625075101852417, -1.4675527811050415, 0.5860894322395325, -0.19566307961940765, 0.35575297474861145, -0.5098885297775269, -0.04056013375520706, 0.5735107064247131, -1.5148543119430542, 0.6180291175842285, -1.378149151802063, 0.34260207414627075, -3.1384456157684326, -0.6962827444076538, 1.7698794603347778, 0.2065889835357666, -0.20770548284053802, -0.14152438938617706, 0.6534771919250488, 0.5008637309074402, -0.7863144278526306, 0.32323598861694336, -0.12104009091854095, 0.3716059625148773, 0.637621283531189, -1.2285405397415161, -0.6704335808753967, 1.20621919631958, -0.6345150470733643, 0.18081125617027283, 0.7185717821121216, 1.1206836700439453, 0.23252753913402557, -1.474841833114624, -0.6448497176170349, 1.8481050729751587, 0.3133224844932556, -1.2396199703216553, 0.1885494589805603, -2.3660759925842285, 0.05698015168309212, 1.514988899230957, -1.034817099571228, 1.2314043045043945, 0.6530599594116211, 1.5907635688781738, 0.48633235692977905, 0.6334145069122314, 1.2443695068359375, -0.3764815628528595, -1.4988558292388916, 1.0959025621414185, 0.5565891861915588, -1.3034170866012573, -1.2594455480575562, 0.3859107792377472, -0.09588389098644257, -1.5114034414291382, -0.3277829587459564, -0.8756092190742493, 1.3865113258361816, 0.5751305818557739, 0.40452107787132263, 0.7452495098114014, -1.3880101442337036, -0.25997892022132874, 0.466123104095459, 0.047534771263599396, -0.4275698959827423, 0.7344645261764526, -0.4914371967315674, 0.21851249039173126, -0.1988968849182129, -0.4446578621864319, -0.13141494989395142, -1.9670350551605225, 0.6613415479660034, -0.5211923718452454, -0.34716376662254333, -1.1603798866271973, -1.4191288948059082, 1.138674259185791, 0.10147832334041595, 0.44302472472190857, -0.4127805233001709, 1.2107144594192505, -1.0038139820098877, 1.3277744054794312, 0.36095401644706726, 1.181544542312622, 1.0987703800201416, -1.1530786752700806, 0.7473394274711609, -1.0739328861236572, 0.3913419246673584, 1.3957619667053223, -0.8581209182739258, -0.5046029686927795, -0.7276320457458496, -0.5539467930793762, -0.21447136998176575, 1.576324701309204, 0.6049455404281616, 1.276978850364685, -0.7283206582069397, 0.9408217072486877, 0.34700602293014526, -0.6139389872550964, 0.2666623890399933, -1.1217877864837646, 1.3715811967849731, -0.31106308102607727, -0.18131877481937408, 0.0847708135843277, 1.0949904918670654, 0.1877007931470871, 0.23159609735012054, 1.1218842267990112, -0.1548985242843628, 0.37276214361190796, -0.2669130563735962, -0.2827264666557312, 0.5123966336250305, 0.7339792251586914, -0.7769384384155273, -0.29125791788101196, -2.0948362350463867, 1.7316254377365112, -0.35425013303756714, -1.731055498123169, 1.0748090744018555, 1.5989359617233276, 0.605551540851593, 1.1371853351593018, 0.9841772317886353, -2.39475679397583, -1.193195104598999, -1.8867722749710083, -0.04482676833868027, -1.1928855180740356, 0.7582014799118042, -0.6106988787651062, -1.1173934936523438, 0.5245676040649414, -0.29760852456092834, -0.9175998568534851, -0.25264856219291687, 0.6590051054954529, 0.1857784539461136, -0.06846702843904495, -0.790624737739563, -0.7908033132553101, -0.09438506513834, -1.0870944261550903, -0.16639287769794464, -1.7649325132369995, 0.14376552402973175, 1.4096983671188354, -0.626564621925354, -0.5584283471107483, -0.1695728600025177, -0.8321416974067688, 0.35007837414741516, 2.205897569656372, 2.053131580352783, -1.5633599758148193, 0.9845894575119019, 0.16978654265403748, 0.48614317178726196, 1.1793609857559204, 0.006984129548072815, 0.46147459745407104, -0.3835470378398895, -0.08848479390144348, 0.05301102250814438, 0.28382283449172974, -0.9111753106117249, 1.0122336149215698, 0.33410945534706116, 1.052093744277954, -0.737042248249054, -2.029247999191284, 0.2223050743341446, 0.052382227033376694, 1.3912469148635864, -0.19843661785125732, -0.6952947974205017, 0.5684048533439636, -1.607170581817627, -0.010267537087202072, -0.5864656567573547, 0.764213502407074, -0.3957308530807495, 0.2263646274805069, -0.3004114329814911, -0.40870755910873413, 1.0725551843643188, -0.3074706792831421, 1.7157695293426514, 0.12885598838329315, -0.5993364453315735, 1.243404746055603, -0.6721706986427307, -1.3091398477554321, -1.1265349388122559, -0.5577964186668396, 0.8733751773834229, 0.5254872441291809, 0.764780580997467, 0.15900924801826477, -0.687728226184845, -1.5341438055038452, 0.02766237035393715, -0.3771013021469116, -0.5732653141021729, -0.9866915345191956, -0.7011315226554871, 0.7420989274978638, 0.6911865472793579, 0.14089523255825043, 1.8417338132858276, -1.3081084489822388, -0.5331527590751648, 1.3294304609298706, -0.9198576211929321, -0.7311820983886719, 0.27446213364601135, 2.126636028289795, 0.8205891847610474, -0.914854884147644, 0.8780831098556519, 0.35525158047676086, -1.5507729053497314, 0.17921742796897888, 1.5509939193725586, -0.8008044958114624, -0.09023234248161316, 0.3727247416973114, 1.340334415435791, 0.9019659161567688, 1.4532802104949951, 1.03843355178833, -0.4729209840297699, 1.5813719034194946, 0.06464511901140213, -0.355425089597702, 0.8538451194763184, 1.7722877264022827, 0.028020154684782028, 1.1453832387924194, 2.1429240703582764, -0.7765446901321411, 0.9851195216178894, -0.8205715417861938, -0.40680041909217834, -0.5451829433441162, -1.0953543186187744, 0.5617753863334656, 0.9522035121917725, 1.2979761362075806, 1.3112584352493286, 0.5224447846412659, -0.06451143324375153, -0.6643707156181335, 0.26347965002059937, 0.1799210160970688, -0.32671496272087097, -1.4071130752563477, 0.9126999378204346, 0.8621973991394043, 1.0655549764633179, 1.0485159158706665, -1.6431483030319214, -0.8473722338676453, 0.7052001357078552, -0.2488952875137329, 0.7531648874282837, -0.10081560164690018, -0.5519858598709106, 1.0185587406158447, 0.17652982473373413, 0.4317496716976166, 2.135599374771118, -2.2859692573547363, 0.9100565910339355, 1.5107704401016235, 0.8690131306648254, 1.427546739578247, 1.1942371129989624, -0.8692002892494202, -0.29925665259361267, -0.7620271444320679, 0.09393056482076645, 2.0440900325775146, 1.2846999168395996, 0.04261961579322815, -0.30713793635368347, 0.19033843278884888, 0.8974832892417908, 0.42829430103302, -3.4212329387664795, 0.04781501367688179, 0.7599783539772034, -1.0758129358291626, -0.49969449639320374, -0.08391997963190079, -0.8777663707733154, -0.8292844295501709, -0.4046272933483124, 0.4397251307964325, -0.44227921962738037, 0.08729341626167297, -0.7427173852920532, 0.21608105301856995, 0.2616290748119354, -1.2178728580474854, -0.5572002530097961, -0.804412305355072, -0.0011214400874450803, 1.1725866794586182, -0.5998609066009521, -0.0134214386343956, 1.1616175174713135, 0.5236772894859314, -0.7075530290603638, -2.568730592727661, 0.5495750308036804, -0.4647355377674103, 1.4536876678466797, -0.4378211796283722, 0.14840039610862732, -2.3164875507354736, 0.065651535987854, 0.35041534900665283, -1.0090522766113281, 0.3473348021507263, -0.13214047253131866, -0.3309030532836914, -2.510932445526123, -0.8281892538070679, -0.9693734645843506, -0.5138100385665894, 1.171797752380371, 0.9503331780433655, -2.699497938156128, 0.37012144923210144, 1.5898951292037964, 0.5776118636131287, 0.8175475597381592, 0.16828800737857819, 1.3731920719146729, 1.604383111000061, 1.5886282920837402, 1.694832444190979, 0.606604814529419, 0.8894996643066406, 0.5589446425437927, 0.7197247743606567, -0.6410009264945984, 0.4829699695110321, -0.2921488583087921, 0.4747004806995392, -1.3034499883651733, -0.24213054776191711, -0.8656792044639587, -0.036387234926223755, 0.2797496020793915, 0.09227427840232849, 0.06894620507955551, 0.02482801489531994, 0.7838189005851746, 0.5452861189842224, 1.5133048295974731, -0.336328387260437, -1.4654244184494019, -2.798628091812134, 1.6591107845306396, 1.4058352708816528, 0.297611802816391, 0.09029582887887955, 0.5043426752090454, 1.0421396493911743, -1.5571074485778809, -1.2840771675109863, -0.006919047329574823, 0.49414628744125366, -0.6115097999572754, 1.060263991355896, 1.324459195137024, -0.06551717221736908, -0.10335429012775421, 1.050558090209961, -0.47460049390792847, -1.3437678813934326, 0.09345860779285431, 0.9868244528770447, -1.7497228384017944, 0.5958423018455505, 0.5651832818984985, 0.8392400145530701, 0.006129777524620295, -1.5811318159103394, 1.6234922409057617, 0.2777685821056366, 0.4567614197731018, -0.7190786600112915, 0.6342915296554565, 0.01145304273813963, -0.22429341077804565, -0.1662004590034485, -0.4866664409637451, -1.0480364561080933, 1.1833776235580444, 0.6855788826942444, 0.5364763140678406, 0.39235273003578186, 0.9906412363052368, -0.13140013813972473, -0.3826304078102112, -0.9025697112083435, -0.6484948396682739, -0.5952550172805786, -1.3839895725250244, -0.41021981835365295, 0.1738593429327011, 0.7016699910163879, -0.7914515733718872, -2.519644021987915, -1.1504508256912231, -0.3094945549964905, 0.1673305630683899, 1.1740983724594116, 1.5514858961105347, 0.5309506058692932, -1.7544690370559692, -0.05907107889652252, -1.7002490758895874, 1.271199107170105, 0.7155216932296753, -0.7960134744644165, 0.08309266716241837, -0.25466519594192505, -0.6813516020774841, -0.3402310907840729, -2.3455469608306885, 0.6081585884094238, 0.6601134538650513, -0.28324541449546814, 0.12185922265052795, -0.007207510527223349, -0.012386612594127655, -1.3800206184387207, 0.5553262233734131, 1.0508852005004883, -1.1604623794555664, 0.2352893054485321, -0.7464497685432434, 0.07474225759506226, 0.6873677372932434, -0.27605655789375305, 0.10158801823854446, -0.20238231122493744, 0.07519605755805969, -0.5435003638267517, -2.3179218769073486, -0.19127100706100464, -1.9815342426300049, -0.7673166394233704, 0.9705824255943298, 1.8398425579071045, -0.3698973059654236, -1.2626380920410156, 0.7122880220413208, -0.43713536858558655, -1.2569223642349243, -1.8361597061157227, 0.6310380697250366, -0.2776152789592743, -1.1928601264953613, 0.05260394513607025, -0.2622312903404236, 0.219563826918602, -0.5996596217155457, 0.8946126699447632, 0.14136512577533722, -0.6321654319763184, -1.0464651584625244, 0.5494983792304993, -0.7743813991546631, -0.647873044013977, -0.9726896286010742, -0.19995862245559692, 0.18890763819217682, -0.3832609951496124, -0.895928680896759, 0.04700920358300209, 1.113533854484558, 0.4003750681877136, 0.6450878381729126, 0.2555701434612274, -0.7273598313331604, 0.8387230634689331, -0.5708934664726257, -0.5716452598571777, -1.5435986518859863, -1.3017994165420532, -0.5207304954528809, 2.7310144901275635, 0.3836502432823181, -1.7413328886032104, 0.616604208946228, 0.5383696556091309, -0.751711368560791, -0.15599465370178223, 1.4866708517074585, 0.751270055770874, 3.108642816543579, 0.6448777914047241, -1.9422557353973389, 1.241999626159668, -1.584166169166565, 2.035238742828369, -0.17016658186912537, 0.12578563392162323, -0.750864565372467, -0.10068793594837189, -0.08476560562849045, 0.2834652066230774, -0.5022103786468506, 0.4520513713359833, 0.9908348917961121, 0.49797725677490234, 0.7596336603164673, -0.5919275879859924, 0.08986839652061462, -1.0934596061706543, 1.939835548400879, 1.1617405414581299, 1.2900418043136597, 1.7220145463943481, 0.223082035779953, -1.7804739475250244, -0.7690911293029785, 0.06856272369623184, -0.4810613691806793, -1.6567106246948242, -0.5518640279769897, -0.1790609508752823, -2.1978461742401123, 0.04930150881409645, 1.0911155939102173, -2.21505069732666, -0.6378047466278076, 1.2156399488449097, 1.2877267599105835, 1.0739113092422485, -0.07766498625278473, -0.8857206702232361, 0.6415492296218872, 1.169668436050415, -0.14661698043346405, -0.33542269468307495, 0.7946053743362427, 1.3432202339172363, -1.208103060722351, 0.7498447895050049, 0.22917352616786957, 0.12667317688465118, -0.6900800466537476, -0.23354309797286987, 1.3767272233963013, 0.6756443381309509, 0.1037738174200058, 0.3390043377876282, -0.8879522681236267, -2.992121458053589, 0.6954595446586609, -1.3212366104125977, 1.692688226699829, -1.0086613893508911, -0.2667801082134247, -0.21768434345722198, 0.7113772630691528, 0.6672631502151489, 0.6278167366981506, -0.40044623613357544, 1.0052881240844727, -0.6700458526611328, 1.2965742349624634, -2.302450656890869, 0.7668070197105408, -1.7546279430389404, 0.41970738768577576, 1.4814319610595703, 0.85037761926651, -0.4482651352882385, 1.077678918838501, 1.436082124710083, -0.7002741694450378, -0.10299967229366302, 0.23917752504348755, -0.2907082438468933, 0.10681302100419998, 1.0053834915161133, -0.8711525201797485, -0.8491744995117188, 0.2561466693878174, -0.41447609663009644, -0.5646980404853821, -0.23591451346874237, -0.32346078753471375, 1.1933002471923828, -0.6671175956726074, -0.9501419067382812, -0.8228051066398621, -0.30695804953575134, 0.7039218544960022, -0.693182647228241, 0.2999262809753418, -1.125389575958252, -1.360883116722107, -1.1084709167480469, 0.6176267266273499, 0.38057342171669006, 0.04402608424425125, -0.7992597222328186, -0.04583779349923134, -1.7454677820205688, -0.14598360657691956, -1.9824974536895752, 1.382989525794983, 0.7566088438034058, -0.5148764848709106, -0.7335246205329895, -0.12475311011075974, 2.096248149871826, 1.2477033138275146, 0.17062535881996155, 0.593562662601471, 0.36640334129333496, -0.0030874451622366905, 1.3904846906661987, 0.18714480102062225, 0.017623985186219215, 1.399665117263794, 1.8616909980773926, -0.722957968711853, 0.5230750441551208, -1.5104626417160034, -0.4991983473300934, -0.33861860632896423, -0.32534894347190857, -0.7543429732322693, -1.4813579320907593, 0.14851878583431244, -0.281295508146286, -1.3505457639694214, 0.8019261956214905, 0.7710338830947876, 1.4223387241363525, -0.48107513785362244, 0.6667094826698303, 0.09520524740219116, 0.03215004876255989, -1.3722102642059326, 0.3603985905647278, -0.06487619876861572, -0.5941177606582642, -0.965153694152832, 0.7834050059318542, -0.14378470182418823, -0.2546384632587433, 0.1944534033536911, 0.39596298336982727, 1.9193966388702393, 0.7530523538589478, 1.8858497142791748, -0.4813462197780609, 1.6045722961425781, 0.6594998240470886, 0.018971193581819534, -1.1851304769515991, -0.45303064584732056, -0.7206239700317383, 0.3994785249233246, 1.647928237915039, 0.04424658045172691, -0.36821818351745605, 1.751142978668213, 0.25304684042930603, 0.6650390028953552, -0.97245854139328, 0.9932205080986023, -0.48868975043296814, -0.5604988932609558, 0.7836690545082092, -0.3119298219680786, 0.03151939436793327, 0.5876116156578064, 1.8975640535354614, -0.5637912750244141, 0.9118022322654724, 0.9898555278778076, -2.176497220993042, 0.7874067425727844, -1.4255183935165405, -0.22896207869052887, 0.050111718475818634, -0.5020380616188049, 0.09202265739440918, 1.3771687746047974, 0.7501022815704346, 1.2766317129135132, -0.42390546202659607, -0.2210226207971573, -0.010036745108664036, -0.5278914570808411, 0.1918487250804901, 0.07132783532142639, -0.36317089200019836, -1.2944306135177612, 0.5740010142326355, -1.0144261121749878, -2.1892552375793457, -0.5066432952880859, 0.15704786777496338, -1.2651108503341675, -0.09910321235656738, -0.8577398657798767, 1.0699397325515747, 0.17737649381160736, -1.7611939907073975, 0.4800091087818146, 0.8786170482635498, 0.11254066973924637, -1.3666555881500244, -0.11241529881954193, -0.7885914444923401, 0.8844009041786194, -0.5366449952125549, -0.5372700095176697, 1.5832974910736084, 0.2895793616771698, -0.40962862968444824, -1.17348313331604, 1.205489993095398, -0.21794326603412628, -1.3549647331237793, -0.8700029850006104, 0.008485605008900166, 0.6381196975708008, 0.07714703679084778, -0.41747763752937317, 1.6101083755493164, -0.6113893985748291, -0.9458027482032776, -0.4431145191192627, 0.14567892253398895, 0.9823729991912842, -0.39511582255363464, 0.1502344310283661, 0.4783799648284912, -1.4909323453903198, 0.8140947222709656, 0.1221420168876648, 0.44814878702163696, -0.5645130276679993, 0.5080972909927368, 0.017670365050435066, 1.8827799558639526, 0.8470237851142883, 2.545424699783325, 0.3828834295272827, 1.9985706806182861, 1.3919965028762817, 1.305726170539856, 0.03566499054431915, 0.8245231509208679, 0.7491738200187683, -0.012701760977506638, -0.6542374491691589, 0.8843138217926025, 1.1947592496871948, 0.6470917463302612, 0.35877135396003723, -1.160393476486206, 0.012697605416178703, -0.39062806963920593, 0.19385294616222382, -1.2477710247039795, 0.12211012840270996, 0.17779172956943512, 0.3309468924999237, -0.6295080184936523, 0.6227421164512634, 0.0414942167699337, -0.40036046504974365, -1.9314467906951904, 1.6751986742019653, 1.4590528011322021, 0.5258931517601013, 0.9359019994735718, -0.7116994857788086, 1.2477140426635742, -0.8999996781349182, 0.563299834728241, -0.18145233392715454, -0.7086440324783325, -0.21052928268909454, 0.11935935169458389, 1.1525812149047852, 0.11989090591669083, -0.36546093225479126, -0.013839914463460445, 0.10032457858324051, 0.07700228691101074, -0.8449597358703613, -0.5583589673042297, 0.12080284208059311, -0.5897527933120728, -1.0176129341125488, -0.5835641026496887, 0.7657119631767273, 0.11957778036594391, 0.7346980571746826, 0.6560962796211243, 1.4031168222427368, -0.27285078167915344, -0.42796817421913147, 0.42059993743896484, -0.24202902615070343, 0.1665925830602646, 0.4442968964576721, -0.4548029899597168, 0.9098270535469055, -0.8626713752746582, 1.0173166990280151, 1.5795316696166992, -1.0340269804000854, -1.425593376159668, -0.5493985414505005, -0.7237278819084167, -0.5359743237495422, -0.74644935131073, 0.8444934487342834, -0.48485222458839417, 0.17696526646614075, 1.1888391971588135, -0.5351189374923706, -0.08244127780199051, -0.020383747294545174, 1.0877605676651, 0.4896746277809143, 2.244872808456421, -0.6413068771362305, 0.31533944606781006, 0.22868883609771729, 0.5088145136833191, 0.16546931862831116, 1.2211813926696777, 1.6474934816360474, -0.20464862883090973, 0.39343908429145813, -0.9514273405075073, 1.162791132926941, -2.082136392593384, -2.3849620819091797, 0.045233577489852905, -0.7209997773170471, 1.452182412147522, 0.6530002951622009, 2.6869242191314697, 1.4859520196914673, -0.08001309633255005, 1.8322803974151611, -0.9829420447349548, 0.6620762944221497, -1.3940973281860352, -0.8411574959754944, -1.113089680671692, 0.8474475741386414, 1.1804442405700684, -0.37628886103630066, -0.7374378442764282, 1.2072808742523193, 0.17719164490699768, 0.5401937961578369, 1.1341971158981323, -0.052816133946180344, 0.17519338428974152, -0.14443248510360718, -0.5285927653312683, -2.4118289947509766, -1.4863654375076294, -1.9851446151733398, -1.304538607597351, -0.6686170101165771, 0.8361412286758423, 0.3609279990196228, -0.4632721543312073, -0.30893683433532715, 0.6983782052993774, -0.34725090861320496, 0.07537813484668732, 0.8201639652252197, 0.35898613929748535, 0.8539906144142151, 0.02046351693570614, -2.097658395767212, 1.9890912771224976, 0.5005504488945007, -2.0534443855285645, -0.23136334121227264, 0.23006916046142578, 2.123774528503418, -2.230461835861206, -0.2588542103767395, 0.18396353721618652, 1.218765377998352, -1.371923565864563, -0.6805916428565979, -1.9505149126052856, -1.1306052207946777, -0.833717405796051, -0.580927312374115, 0.550717294216156, -1.9882594347000122, 0.12478902190923691, -1.4106148481369019, 0.5814394950866699, 0.8920726776123047, -0.2675340473651886, -0.9185286164283752, 0.47467613220214844, 0.7317219972610474, -1.0817203521728516, -1.1070972681045532, -0.3469468653202057, 0.11761828511953354, -1.3037443161010742, -0.012308517470955849, -0.22105169296264648, -0.9129751920700073, 1.0566372871398926, -1.7839086055755615, 1.601530909538269, -1.5101982355117798, 0.08608736842870712, 1.3436111211776733, 0.37195926904678345, -1.0993353128433228, 1.0200059413909912, -0.28189781308174133, 1.683616042137146, 0.005510780028998852, 0.4184207618236542, -0.994422972202301, -1.114119052886963, 1.138417363166809, -1.8515864610671997, -0.5258166193962097, -0.5958212614059448, -0.5556356310844421, -0.6252886056900024, -1.7777210474014282, -1.6456806659698486, -0.38575097918510437, 0.45706358551979065, -1.034190058708191, -0.0742640569806099, -1.3301440477371216, 0.4298340976238251, 0.5110859274864197, -1.8153388500213623, -0.36632129549980164, 0.5502000451087952, 2.403978109359741, -1.104454517364502, 0.4361002445220947, -0.08744160085916519, -0.5426060557365417, -0.15360590815544128, 1.2386153936386108, 0.2586694061756134, 0.5026590824127197, 0.40510013699531555, -2.1382362842559814, -1.2406502962112427, -0.8260986804962158, 0.16978487372398376, -0.13489793241024017, 0.4501854181289673, -0.2970629036426544, 0.3837868869304657, 0.4147273898124695, -1.1586041450500488, -0.7114948034286499, 0.04097157344222069, 1.2386436462402344, 1.1199334859848022, -0.67472904920578, -0.32273849844932556, 0.19723521173000336, 0.5785463452339172, -0.49493643641471863, -0.274724543094635, -1.6225403547286987, -0.5587322115898132, 0.9437151551246643, -0.10646801441907883, -1.6504279375076294, 0.7823618054389954, -0.9818642735481262, 3.310119390487671, -0.6307857632637024, 1.135608196258545, -1.3004902601242065, 1.1346462965011597, 0.3134506940841675, 2.435943841934204, -1.6210763454437256, 0.7750317454338074, -0.06616856157779694, 0.31129786372184753, -0.004991600755602121, 0.08747706562280655, 0.5658838152885437, 0.293392539024353, 0.2804363965988159, -0.5966599583625793, 0.7095080614089966, 0.03726213797926903, -1.4513124227523804, 0.22397764027118683, -0.20884941518306732, -0.024011356756091118, -0.16291280090808868, 1.5670987367630005, -0.7347511649131775, 0.4795500338077545, 1.8617249727249146, -0.29318323731422424, 0.5841516256332397, -0.2956234812736511, 1.2985725402832031, 0.7441346645355225, -0.36096733808517456, 0.7071971893310547, 0.3199399411678314, -0.9065327048301697, -1.4835714101791382, -1.077397108078003, 0.4038601219654083, -0.403666228055954, -1.8963873386383057, -0.534911036491394, 0.2737133204936981, -0.8724087476730347, 0.6075047254562378, -1.9304579496383667, -1.0015687942504883, -0.6000599265098572, 1.231671929359436, 0.7761242389678955, 0.5939565300941467, 0.5240598320960999, 0.8569045662879944, -0.1738724708557129, -0.39657604694366455, 0.5988650918006897, -0.2960626184940338, -1.565375804901123, 1.2364684343338013, -0.5055371522903442, -1.152841567993164, -1.3545292615890503, -0.5419256091117859, 1.593464732170105, -1.1045037508010864, -1.1883831024169922, 0.7460948824882507, 0.00823892466723919, -0.9658061861991882, -0.17529956996440887, 0.0634506568312645, 1.0978301763534546, -0.6152681708335876, 1.1112875938415527, -0.6002233028411865, 0.3522648811340332, 0.3528572618961334, -0.9616145491600037, 1.2822879552841187, -0.818074107170105, -1.27687668800354, -0.8851766586303711, -0.6361391544342041, 2.051476001739502, 1.0704292058944702, -1.6376805305480957, -0.37065985798835754, -0.559331476688385, -1.4530218839645386, 0.6396057605743408, -1.328797698020935, -1.5077952146530151, 0.8179222941398621, -1.5157992839813232, 0.24057932198047638, -0.6361174583435059, -1.4018093347549438, -0.3915804624557495, -0.7121078968048096, -0.05963127315044403, 0.31447461247444153, 1.559714436531067, 0.049247682094573975, 0.12590619921684265, 0.14851665496826172, 0.6620479226112366, 1.068089246749878, -1.1141756772994995, 0.32873955368995667, 0.010425101965665817, 0.22394230961799622, 0.36122697591781616, -1.3961941003799438, -0.1434614360332489, -1.5288989543914795, 1.5450326204299927, -0.31877151131629944, 1.7618294954299927, -0.604347288608551, 0.6753048896789551, 1.7649328708648682, 0.7206171154975891, 0.6900450587272644, 0.37009724974632263, 0.10398209095001221, -0.17316487431526184, 0.08595752716064453, 0.1327652484178543, -0.9653547406196594, -2.487201690673828, -0.18072064220905304, -0.19877448678016663, 0.3244260847568512, -0.8398619890213013, -0.9027517437934875, -0.4924975633621216, -1.5813509225845337, 0.5953918695449829, 0.9750010967254639, -0.2812853157520294, -1.0577861070632935, -1.4689581394195557, -0.26662617921829224, -0.681400716304779, 0.7029135227203369, 0.5251410007476807, 0.20516525208950043, -1.1502594947814941, -0.22031062841415405, -1.416975498199463, -1.0116631984710693, 1.1715861558914185, 0.05868781358003616, 0.3409764766693115, 0.5496542453765869, -0.3599649667739868, -0.3591685891151428, -0.18465714156627655, 0.49089524149894714, -0.18822674453258514, 0.7591935396194458, -1.4277546405792236, 0.838206946849823, -2.002664804458618, 0.06435609608888626, 0.854485034942627, -0.06694750487804413, -0.8477365374565125, -0.94488126039505, 1.1747612953186035, 1.4011160135269165, -0.1422281265258789, 1.5057097673416138, 0.06330577284097672, 0.14197199046611786, 2.1604342460632324, -1.3807297945022583, 0.6129143238067627, -2.3385941982269287, 0.4373323917388916, -1.5382033586502075, 0.16877411305904388, -1.536277174949646, -1.3019700050354004, -1.1809911727905273, 1.0904152393341064, -0.0021316781640052795, 0.22544792294502258, -1.5710084438323975, -0.23384463787078857, -1.5413134098052979, 0.7984206676483154, 1.281368613243103, 1.0482590198516846, 0.3922322690486908, 2.084742546081543, 0.6237481236457825, 0.8406226634979248, -1.4784399271011353, -0.45556139945983887, 2.8901302814483643, 0.016349881887435913, 0.502238392829895, -0.3328437805175781, 0.8084891438484192, 0.21715517342090607, -1.0653142929077148, -0.2979438602924347, 0.11966677755117416, 0.3207795023918152, 1.3132940530776978, -1.948182463645935, -1.5104461908340454, -1.2822561264038086, -1.3444815874099731, 0.4749263525009155, 0.6680752635002136, -0.5295220613479614, 0.8399605751037598, -2.324406862258911, 0.4684669077396393, 1.9570822715759277, 0.6340332627296448, -0.4031696319580078, 0.06714287400245667, -0.630072832107544, -1.456620454788208, -1.8842793703079224, -0.3649572730064392, -0.3491116464138031, -0.3063792288303375, -0.6882309317588806, 1.142248511314392, -0.05589040368795395, -0.8188788294792175, 1.003578543663025, 0.5532726049423218, 0.6704634428024292, 2.240738868713379, 1.4723190069198608, -0.01708502508699894, 0.37638556957244873, 0.10540731251239777, -0.07833161950111389, 0.3307512700557709, -0.49703705310821533, -0.060357898473739624, 0.25312188267707825, -0.29513731598854065, 0.41676998138427734, -0.08641765266656876, -0.22789083421230316, 1.3217231035232544, 1.8180345296859741, 0.5893059968948364, -0.23195390403270721, 1.0899419784545898, -0.450121134519577, 0.7603614926338196, 0.06708860397338867, -0.7842013239860535, -0.06408582627773285, -0.5592871904373169, 2.05851411819458, 1.5239039659500122, -0.8445435166358948, -2.086277484893799, 1.2267694473266602, 0.6909587979316711, 0.9036690592765808, -0.00685783289372921, -0.2569316327571869, -1.122446894645691, -0.8403761982917786, 0.8805941939353943, -0.8990709185600281, 0.40554293990135193, 0.7360326051712036, 1.0099812746047974, -0.4112754166126251, -1.2550454139709473, 1.1616450548171997, -1.4400391578674316, -1.7917406558990479, -0.13378378748893738, -0.3267812132835388, 0.5709865093231201, -0.5073086619377136, -1.4836671352386475, -0.7493442893028259, 0.6911138892173767, -0.7669323086738586, -0.46955183148384094, -0.790306568145752, -1.1927824020385742, 0.5862423181533813, 0.9134898781776428, -0.3246322572231293, 0.012210444547235966, -0.6264013051986694, -0.24569901823997498, 0.3323632776737213, 0.8282225728034973, -0.5036917328834534, -1.5965642929077148, -0.5983411073684692, 0.14856475591659546, -0.3668324053287506, -0.3687993586063385, 0.3742453157901764, -0.650061309337616, 0.3362393081188202, 0.3790963590145111, -0.3216783106327057, -0.33815768361091614, -0.8907995820045471, 0.11775535345077515, 0.8021708726882935, -1.307798981666565, -0.23366788029670715, 0.42306986451148987, -0.20400230586528778, 0.5097262859344482, 1.0556758642196655, 0.2919315993785858, 0.8463178277015686, 0.7919778227806091, 0.9356886148452759, 0.7936428189277649, -1.057032585144043, 1.4561811685562134, 1.294731855392456, -0.6424806118011475, -1.6232718229293823, 0.10877793282270432, -0.4487810432910919, 0.5361326336860657, 1.6886383295059204, 0.43066444993019104, -2.34053111076355, -1.5524098873138428, -0.01585819013416767, 1.5973490476608276, -0.35207703709602356, -0.7422735691070557, 0.3265874981880188, -0.8867858648300171, 0.4563312828540802, 0.3315320909023285, 1.268584966659546, 0.4943777322769165, -0.147700697183609, -0.4326031506061554, -1.1095260381698608, 1.5305007696151733, 0.17090782523155212, -0.4813956320285797, -1.3934326171875, 2.1744556427001953, -2.161776542663574, -0.1649457812309265, -0.11364132910966873, -1.2470133304595947, 0.38001689314842224, 2.3760616779327393, -0.5628837943077087, -1.3996843099594116, 2.0060606002807617, -0.33651721477508545, -0.33622896671295166, -2.3421802520751953, 0.16664431989192963, 1.4612678289413452, -1.1102455854415894, -1.2687230110168457, 1.3581591844558716, -1.2795960903167725, -2.1183454990386963, -0.4797186851501465, 0.0795329138636589, -0.27752014994621277, -0.6428732872009277, 0.10322925448417664, -1.4527591466903687, -0.14346711337566376, 0.612503170967102, 0.7194728851318359, -1.4344781637191772, -1.0086629390716553, 0.10666171461343765, -0.44006288051605225, -0.4305388927459717, -1.3498412370681763, -1.136574149131775, 2.0598809719085693, -0.28248679637908936, -1.49087393283844, -0.578168511390686, -1.4399938583374023, -0.5780346989631653, -1.3330494165420532, 1.505164623260498, -0.7854952216148376, -0.29149705171585083, -1.3049228191375732, -1.8450919389724731, 0.03535309061408043, -0.3362812101840973, -1.4823520183563232, 0.7915396690368652, -0.5446182489395142, 0.06360198557376862, -0.3729158639907837, 0.33130189776420593, -1.673360824584961, 0.7150195240974426, 1.02434241771698, -0.5562213063240051, 1.1275858879089355, 0.7315863370895386, 0.0954134538769722, -0.059713903814554214, 0.4334147274494171, -1.8359979391098022, 2.502645969390869, -0.36206313967704773, -1.477365255355835, 1.9101098775863647, 0.20602135360240936, -0.6660966873168945, -0.34455105662345886, -0.9406775832176208, 0.18987467885017395, -1.7927783727645874, -0.08316440880298615, -1.6469812393188477, -0.18471650779247284, -0.24098087847232819, -0.5373838543891907, -0.4876219928264618, -0.39758986234664917, -3.0690758228302, 0.6102802753448486, -1.151594877243042, -0.5600746273994446, 0.13152191042900085, -0.9272811412811279, 1.5302557945251465, 1.984817385673523, -0.8747068047523499, 0.14186254143714905, -0.9610219597816467, -0.2651095986366272, 1.0622435808181763, -1.9057503938674927, 1.4615215063095093, 0.8293230533599854, -0.9047771692276001, 0.5338761210441589, -0.3656839430332184, -0.14316228032112122, 0.2871588170528412, -0.07696720212697983, 1.3649868965148926, -0.4564433991909027, -1.413761854171753, -0.11738424003124237, -0.9764553904533386, -0.9330662488937378, 1.021257996559143, 0.0015796850202605128, 0.33087921142578125, -0.013315561227500439, -0.37357786297798157, -0.45864230394363403, 0.39643925428390503, -0.5995174646377563, 0.34414365887641907, -0.018605515360832214, 2.496556282043457, -0.30691489577293396, -0.7714008092880249, 1.7848641872406006, 0.4391603171825409, -0.3054845631122589, -0.05124182626605034, 1.2097147703170776, -0.539519727230072, -0.13636717200279236, -0.5018957853317261, 0.7437021732330322, 0.40573275089263916, -0.1622331142425537, 1.704980731010437, 2.1995279788970947, -0.6218117475509644, -0.21827608346939087, -1.0489386320114136, -0.6689591407775879, -1.2277098894119263, 0.2736539840698242, 0.01503793802112341, -0.046349771320819855, 1.7675882577896118, 1.1958622932434082, 0.16040383279323578, 0.16785898804664612, -0.3062271773815155, -0.46161016821861267, -0.09664084762334824, 0.15989923477172852, 0.9043205976486206, -0.6703656911849976, -0.3125160336494446, -0.12400082498788834, 1.68770432472229, -0.2968991696834564, -0.8403446674346924, 0.8743818402290344, 1.451749324798584, -0.14672566950321198, -1.4154506921768188, 2.5459179878234863, -1.0693175792694092, -0.7389262318611145, 1.194689154624939, -1.375950813293457, -0.08836499601602554, -0.5597739219665527, -1.8095293045043945, 0.5820865035057068, 0.39756178855895996, -1.5033783912658691, 1.7037346363067627, 0.592284619808197, 0.014026333577930927, 0.5225734114646912, 1.7670634984970093, -0.012763136066496372, -0.2023480385541916, -0.34018146991729736, 0.9368304014205933, -1.2933237552642822, -0.6056808829307556, -0.16873770952224731, -2.338472604751587, -0.7013449668884277, -2.4209518432617188, 0.7970129251480103, -1.1335393190383911, 1.271130919456482, 1.4383314847946167, 0.9090259075164795, -2.310885429382324, 0.3422660827636719, -0.8818673491477966, 0.6781951785087585, -0.9702664017677307, -0.7998844385147095, -0.41859495639801025, 2.240325927734375], [1.1452178955078125, -0.6046034097671509, -1.268493413925171, 0.0731530711054802, 0.3365567922592163, -0.16404001414775848, -0.9449558854103088, -0.31274232268333435, -0.426874041557312, -0.12955304980278015, -0.7519691586494446, -0.2886779308319092, 0.15228275954723358, -0.7798171043395996, -0.2978106141090393, 0.17784643173217773, 0.7470322847366333, 0.5393469929695129, -0.12628090381622314, -0.1716882437467575, 1.0830771923065186, 1.3226243257522583, 1.5072311162948608, -1.2634673118591309, -0.018848180770874023, -0.46004343032836914, 0.14849606156349182, -0.9276464581489563, -0.36740174889564514, 0.5201321244239807, 0.34783273935317993, -0.6614464521408081, -2.128905773162842, -0.09392844140529633, 0.717112123966217, 1.0179753303527832, -2.47629451751709, 0.365894615650177, 0.7023732662200928, -0.1577511727809906, -1.1367300748825073, 0.7899872064590454, 0.11897379904985428, -0.7865482568740845, 1.3650929927825928, 2.005868673324585, -0.33259186148643494, -0.01707797311246395, 2.1693480014801025, -1.5477311611175537, 0.021708820015192032, 0.6444953680038452, -1.5599431991577148, 0.6709442734718323, -0.3727632761001587, -1.0097001791000366, -0.4462106227874756, 1.0348812341690063, 0.31950390338897705, -1.6279773712158203, -0.046902235597372055, 0.7149558067321777, 0.3762996196746826, -0.4102928638458252, -0.5896907448768616, -2.2053000926971436, 2.762385368347168, 0.0022579070646315813, 1.410171627998352, 1.5536725521087646, 1.4066067934036255, 1.2586547136306763, 0.1874522715806961, 1.299149990081787, 1.0654613971710205, -1.450172781944275, 1.3108769655227661, 0.754112720489502, -1.3221756219863892, -0.15807145833969116, -1.2070715427398682, -0.21510770916938782, -0.14612939953804016, -0.11172886937856674, 1.5264900922775269, 1.1040624380111694, -0.5569617748260498, 2.423218011856079, 1.1847790479660034, 0.8380447030067444, 1.7736862897872925, 0.12148953229188919, -1.075764775276184, -0.8513904809951782, -0.7644315958023071, -1.1824651956558228, -1.4931025505065918, -1.4736711978912354, 1.094684362411499, -1.7401775121688843, -0.47287800908088684, 0.04188505560159683, -0.2106347531080246, -0.463632196187973, 0.828623354434967, -0.5929980278015137, 0.6615411639213562, -0.146682471036911, 0.29624950885772705, -2.168800115585327, -0.38546809554100037, -1.9604653120040894, -0.952696681022644, -1.5547560453414917, -0.11180509626865387, 0.22155137360095978, 1.3351622819900513, 1.4825383424758911, 1.3186949491500854, -0.5925034880638123, -1.4815237522125244, -0.9239480495452881, 0.29925569891929626, 0.2068372368812561, 0.15469947457313538, -0.4492928981781006, 0.7077914476394653, -0.1708642840385437, -0.2906990349292755, -1.7041575908660889, 0.7432942986488342, -1.0313960313796997, 0.9845887422561646, 0.334960013628006, -0.2549646496772766, -1.0225993394851685, -0.2543301284313202, 0.245849609375, -0.30022895336151123, 0.646591305732727, 0.8886972069740295, 0.6356196403503418, 1.1815235614776611, -0.30945470929145813, -2.127032995223999, 0.3421940505504608, -0.35323598980903625, -0.1708744615316391, 2.2277743816375732, -0.21824800968170166, -0.4960176348686218, -0.0021902467124164104, -0.09728017449378967, 0.8182178139686584, 0.5630285739898682, 0.33001819252967834, 0.21976231038570404, -0.42800527811050415, 0.6990275382995605, -0.35018467903137207, 0.9119832515716553, -0.6083500385284424, -0.6411836743354797, -0.42592787742614746, -1.7141382694244385, -0.7866376042366028, 0.2049066722393036, -0.27561768889427185, 0.25825026631355286, -1.6681245565414429, -0.26234129071235657, 0.6596291065216064, 1.8186403512954712, -0.7281571626663208, -1.702578067779541, 0.5213741660118103, 1.0264710187911987, 0.9950726628303528, 0.5988339185714722, -1.0675653219223022, -0.0012360054533928633, -0.7488465905189514, -0.08958850800991058, -0.10231531411409378, 0.21539129316806793, 0.5939937829971313, -0.4626254141330719, 1.1511356830596924, -0.6787617206573486, -0.22504734992980957, -0.2722097337245941, 2.3541171550750732, -0.09786020219326019, -1.0080170631408691, -0.2957630455493927, 1.6596758365631104, -0.18454235792160034, 2.0587315559387207, 0.2523703873157501, -0.8597745895385742, 0.5295608043670654, 0.2786932587623596, -2.147806167602539, 0.8473460078239441, 0.14856071770191193, 0.41632503271102905, -0.956938624382019, 0.8341565728187561, -0.4222724735736847, 1.0159037113189697, 1.385864019393921, -0.519953727722168, -0.420070618391037, -0.22828589379787445, -0.2852270007133484, 1.3949851989746094, -1.3785618543624878, 1.0592892169952393, -0.25486764311790466, 1.2686903476715088, 0.2338159680366516, -1.3613818883895874, -0.2250787913799286, 1.037369728088379, 1.1451208591461182, 0.27468594908714294, 1.1456660032272339, 0.5735080242156982, -0.08701369166374207, -0.4737960994243622, 0.42356279492378235, 0.3590742349624634, -0.19047358632087708, 0.192996546626091, 1.5781868696212769, 1.7374300956726074, 1.274172067642212, -1.7521312236785889, -0.21270480751991272, -0.2940930128097534, 0.3220684230327606, 0.5459003448486328, 0.4885435104370117, 0.7239152193069458, -0.9835581183433533, -1.0586695671081543, 0.6681248545646667, -0.7754948139190674, -0.8177216053009033, 1.0477772951126099, -0.25085991621017456, 0.6884183883666992, -0.08245734870433807, -0.2250945121049881, 0.24670495092868805, -0.9572525024414062, 2.73376202583313, -0.22212091088294983, 0.17649458348751068, -0.7295886278152466, 1.9725475311279297, -0.06846993416547775, 0.8179891705513, -1.2252696752548218, -0.3593955636024475, -1.4680306911468506, -1.3695838451385498, -0.7992797493934631, 1.0608952045440674, -1.5971853733062744, 1.0326956510543823, 0.6434489488601685, 0.33024534583091736, -1.246475338935852, -1.2539304494857788, -0.5154871940612793, 0.23013949394226074, -0.83946293592453, -0.3521440923213959, 0.1085369884967804, 0.25229379534721375, -0.8597191572189331, -0.30706238746643066, 1.102229356765747, -1.9988059997558594, -1.097338080406189, -1.228681206703186, -0.22385869920253754, -1.2220264673233032, 0.32310810685157776, -1.1065713167190552, 0.34895986318588257, 0.9189894199371338, -0.31022652983665466, -1.2550908327102661, 1.8493174314498901, -1.5004827976226807, -0.7643995881080627, 1.1443885564804077, -1.894593596458435, 0.5784013867378235, 0.19754479825496674, -0.6594420075416565, -1.261254906654358, 0.0814867913722992, -1.3931024074554443, 0.2249298244714737, 0.22488948702812195, -0.09391684830188751, -0.7344865798950195, 1.3586593866348267, -1.4213181734085083, 1.4557112455368042, -0.0382123701274395, -1.4816980361938477, 0.044414252042770386, 0.5802447199821472, -0.16007764637470245, 0.8204681277275085, 0.16071660816669464, 1.0066397190093994, -0.36247900128364563, -0.7604807019233704, 1.3911075592041016, 0.1408984661102295, 0.09545549005270004, 0.05981402099132538, 1.417431116104126, 0.9184543490409851, 0.6019391417503357, -0.05350399389863014, 0.722694993019104, -0.0508117638528347, 0.49201643466949463, 1.119308590888977, 0.47737932205200195, 2.5988857746124268, -1.0055540800094604, 0.0714055597782135, 0.7916795015335083, -1.2215301990509033, -1.5171177387237549, -0.5959641337394714, -0.0018474257085472345, -0.17467769980430603, -0.37969666719436646, -1.3323014974594116, 1.250921607017517, -0.23010116815567017, -1.478191614151001, 0.24666312336921692, 0.009413143619894981, 0.6554012298583984, 0.8212466835975647, -0.15934011340141296, -0.12472525984048843, -2.1547420024871826, 1.5121901035308838, -0.13078588247299194, -1.668478012084961, -0.4869450330734253, -1.0184615850448608, 0.07017797976732254, -0.7734919786453247, 0.00990491546690464, -0.3733600378036499, 0.8179795145988464, 1.2354029417037964, 0.17474043369293213, 0.5529373288154602, -0.06397057324647903, -1.2509061098098755, 0.3582249879837036, -0.036807429045438766, 0.5783501267433167, 0.07382632791996002, -1.6732062101364136, -1.2891300916671753, -2.85638689994812, -0.42420482635498047, 1.2551887035369873, -0.03423944115638733, -0.8947129845619202, 0.28448766469955444, -0.3849640488624573, 0.3349171578884125, -0.4866589307785034, 0.5694669485092163, 0.35777246952056885, -0.2449384331703186, 0.0038981465622782707, 2.1826565265655518, -0.359998881816864, -0.004507602658122778, -0.055408671498298645, 0.11975645273923874, -0.6189418435096741, 0.10957090556621552, -0.5270874500274658, -1.3173925876617432, 1.8675451278686523, 0.5300202965736389, 0.19497106969356537, 0.8882765173912048, 1.1124073266983032, 0.06678206473588943, 0.514846682548523, 1.9356744289398193, 0.024028562009334564, -2.4116721153259277, 1.0722393989562988, 0.11503078043460846, -0.13468123972415924, -0.90449059009552, -0.28686007857322693, 0.6805939078330994, -0.13274648785591125, -0.05172709375619888, -0.7578619718551636, 0.04702366888523102, 0.4247952997684479, 0.06526513397693634, 1.0734567642211914, -0.8602099418640137, -0.5926278829574585, -0.8136392831802368, -0.1921987682580948, 0.08555040508508682, -0.1579159051179886, -1.1457066535949707, 0.15852269530296326, 0.1516464203596115, -0.7230560183525085, -0.2870170474052429, 0.5624348521232605, -1.0241755247116089, 0.9237300753593445, -0.39732831716537476, 0.5409498810768127, 0.9769793748855591, -1.256032943725586, -0.6813608407974243, 1.1517493724822998, -0.8672143220901489, -1.371992588043213, 0.7651930451393127, -1.0258203744888306, 0.5909861922264099, 0.5438746213912964, -0.42096835374832153, 1.0633752346038818, -0.7038502097129822, 0.776374876499176, 1.5688601732254028, 1.5197358131408691, 0.5939474701881409, 0.8874014019966125, 0.9599158763885498, 0.15716855227947235, -1.0690633058547974, -0.47264426946640015, -0.924324631690979, 0.7867761850357056, 0.32172641158103943, 0.8482011556625366, -0.882118821144104, 0.42984578013420105, -0.9338574409484863, -2.1541731357574463, 0.20457503199577332, 1.379225254058838, -3.7031726837158203, -3.112924098968506, -1.0705132484436035, -0.7362213134765625, 0.023462165147066116, 0.38698020577430725, 1.1014727354049683, -0.1419520527124405, -0.92257159948349, 1.349621295928955, -1.2940723896026611, 1.0652375221252441, 1.944770097732544, -0.38659408688545227, -0.09205581247806549, 1.00679349899292, 0.24011020362377167, 0.014854715205729008, 1.081992268562317, -1.0798240900039673, 0.6286548972129822, 1.0069656372070312, 0.7596046328544617, -2.0104732513427734, 0.19125768542289734, 0.004842559341341257, -0.6928930878639221, 0.7393896579742432, -0.2333735078573227, 1.3745174407958984, 0.15254616737365723, 0.30512356758117676, -0.24763379991054535, 1.5046383142471313, 0.4923216998577118, 1.831254005432129, 0.6489397883415222, 0.6059462428092957, -0.40247127413749695, 0.4668494760990143, -0.32301071286201477, -0.4243977665901184, -1.291671872138977, 0.184217631816864, -0.05455869808793068, -0.46863633394241333, 0.4016934037208557, 0.7367655634880066, -1.10201895236969, 0.5877482295036316, 1.7620000839233398, -0.2256546914577484, -0.7657820582389832, -0.8265511989593506, 0.714028000831604, -0.7775720357894897, -0.7644953727722168, 1.0068665742874146, -0.9407868981361389, 0.6684675216674805, -1.0447300672531128, 1.148683786392212, -0.2970191240310669, -0.7671941518783569, 0.5603697896003723, -0.11315629631280899, 0.8379868865013123, 2.091097354888916, 0.5526773929595947, 0.3617420196533203, 1.079246163368225, 0.8863335847854614, -0.42650845646858215, 0.9829777479171753, -0.06418411433696747, 0.807595431804657, -0.10102086514234543, 1.972442388534546, -0.4156630039215088, -0.023105286061763763, 0.23709726333618164, 0.9683583378791809, -1.0353516340255737, -1.17115318775177, -0.1533655822277069, 0.03374563530087471, -0.9950525164604187, 0.2335788458585739, 0.09655861556529999, 0.10512381047010422, 1.7735965251922607, 0.3663998544216156, -1.2418917417526245, -1.292385458946228, -0.4500149190425873, -0.29755333065986633, -1.0901695489883423, 0.7470571994781494, 1.406997799873352, -0.7109938859939575, -0.9692603349685669, 1.6967110633850098, -0.10538223385810852, -0.10594629496335983, 0.4334874153137207, -0.7446544766426086, 0.5632137656211853, 0.6719692945480347, 0.013364638201892376, -1.0831687450408936, -1.1458749771118164, 0.787783145904541, 0.027423372492194176, 0.1269170492887497, 0.4427318274974823, -0.054950982332229614, -1.6517212390899658, -1.0817646980285645, 0.22381165623664856, 0.09410945326089859, -0.8415115475654602, -1.1446985006332397, 1.3089765310287476, 0.3013949990272522, -0.6502792835235596, -0.13660724461078644, -0.45745429396629333, -0.13926009833812714, 1.0833314657211304, -0.330716997385025, -0.49370890855789185, -1.2091952562332153, -0.2867666184902191, -1.402031660079956, -0.08005684614181519, -2.6412596702575684, -1.23215913772583, -0.20421892404556274, -0.6782969236373901, -0.29931527376174927, -0.9494372010231018, -0.5528386235237122, -0.8483449220657349, 0.026079580187797546, -1.5318799018859863, 1.4982061386108398, 1.473644733428955, -0.03146718069911003, 0.6161851286888123, 1.3628093004226685, 1.3524588346481323, 1.6096255779266357, 0.04601434990763664, 0.26312440633773804, -0.4520890414714813, -1.734993577003479, 0.6464712023735046, -1.1628295183181763, -1.457460641860962, -1.3259915113449097, -1.0446971654891968, -0.7231947183609009, -1.274656891822815, -0.5055981278419495, -2.6370198726654053, 1.2918840646743774, 0.5856418609619141, -0.7843146920204163, 1.824620246887207, 0.8046188354492188, -0.9549745321273804, 0.7944313883781433, 0.5541629791259766, -2.491603374481201, 0.9831651449203491, -0.8630546927452087, 0.0704813003540039, 1.8139704465866089, 0.6835512518882751, -0.5457575917243958, -0.3775565028190613, -0.27155357599258423, -0.14629718661308289, -0.4415219724178314, 0.8332811594009399, 0.18647009134292603, 0.009897849522531033, -0.9000759124755859, 1.9565621614456177, -0.5824015736579895, -1.2711020708084106, 0.28089672327041626, -0.6456872224807739, -2.0243661403656006, -1.2520936727523804, -0.8006372451782227, 0.47984805703163147, 0.007884226739406586, -1.4226027727127075, -1.6669132709503174, 0.8756445646286011, -0.9679656624794006, -1.495640754699707, 0.7356069087982178, 0.7497707009315491, 1.2896690368652344, -0.5121085047721863, -0.4432617723941803, -0.8222812414169312, -1.940043330192566, 0.6301376819610596, 0.12779954075813293, 0.7354149222373962, 1.4893295764923096, 1.4957696199417114, 1.1103242635726929, -0.3963530361652374, 0.6092407703399658, -0.8706428408622742, 0.6028882265090942, -0.7093139886856079, 0.37811750173568726, -0.10369831323623657, 0.012069463729858398, 0.2745820879936218, 0.42423015832901, -0.3839723765850067, -0.12354867905378342, 0.24421797692775726, -0.5262555480003357, 0.03113393671810627, 1.0805226564407349, 0.9053196907043457, 2.0027663707733154, 1.3850992918014526, -0.38295984268188477, 0.2042035460472107, 2.5882318019866943, -0.8334721326828003, -0.6900151371955872, -0.35712435841560364, 0.7322287559509277, -0.08104772120714188, 1.333805799484253, -0.7138695120811462, -0.23295003175735474, -0.5107207298278809, -0.22091467678546906, -0.8656690716743469, 0.12858524918556213, -0.5495495796203613, -0.5106630325317383, -0.9783462285995483, 0.06694459915161133, -1.4534579515457153, 0.9382616281509399, -0.2791825234889984, -0.13565972447395325, 1.995249629020691, 1.031295895576477, -1.6892391443252563, 0.10074172168970108, -0.19239750504493713, -1.082962155342102, 0.5927470326423645, 0.37847578525543213, -0.5471111536026001, -1.0487825870513916, -0.454250693321228, -2.721632719039917, 0.033687151968479156, -1.0826977491378784, -1.6826722621917725, 0.5655401349067688, 0.09900596737861633, -0.7110495567321777, 0.9330273270606995, 1.3762845993041992, -0.9744634628295898, 0.22121360898017883, 0.9521251320838928, 0.287679523229599, 0.8524163961410522, 0.02943742461502552, -1.4934004545211792, 0.17375212907791138, 0.542056143283844, 0.8898854851722717, 0.14150789380073547, -1.484745740890503, 0.20109418034553528, -1.422230839729309, 0.39174994826316833, 0.03771727532148361, 0.1857791692018509, -0.0760960504412651, -0.0016119714127853513, -1.3614623546600342, -0.08378693461418152, -0.3473484218120575, -0.18481487035751343, 0.044244103133678436, 1.4446057081222534, 1.4323710203170776, -0.3457964360713959, -1.2954950332641602, -1.631691575050354, 1.4250621795654297, -0.15565982460975647, 0.05255608260631561, 0.08945631980895996, -0.16732940077781677, 1.0232453346252441, 0.2771759629249573, 0.7810353636741638, 1.0327962636947632, -0.661472737789154, -0.04317331686615944, 2.2439162731170654, 0.6705812811851501, -1.388493537902832, 1.4499804973602295, -0.601147472858429, -0.9874503016471863, 0.2996407747268677, 1.4297475814819336, 1.9412838220596313, 1.7215170860290527, 0.4672800898551941, -2.248286724090576, -1.0004998445510864, -1.0396440029144287, -0.6924549341201782, -0.172826886177063, -1.0138615369796753, -0.8143513202667236, -0.4059363305568695, 0.8629305958747864, -0.22955766320228577, -0.9099357724189758, -0.4940492510795593, 0.07720275223255157, 0.7779242396354675, 0.9010859131813049, -0.9568195939064026, -1.2260428667068481, -2.4780986309051514, -0.9035654664039612, 0.5067611336708069, 0.03583403676748276, 0.997151792049408, 1.03319251537323, 0.38426342606544495, 1.5450917482376099, -0.16399507224559784, 1.335902452468872, 0.8964245319366455, -1.1889210939407349, -0.638163685798645, -0.6027069091796875, -0.27035248279571533, 0.49765148758888245, -1.4500226974487305, 1.500295639038086, 1.4903466701507568, 0.49410131573677063, 0.4415871202945709, -0.586555004119873, 0.01788657158613205, -2.64207124710083, 0.32236069440841675, -0.8300395607948303, -0.10823402553796768, -0.6395877003669739, 0.08532635867595673, -0.4405486285686493, 1.5272644758224487, -1.0890495777130127, 0.4238586723804474, 0.07768350839614868, 0.9362353086471558, -1.7671046257019043, 0.11484896391630173, -0.6331137418746948, 1.2049304246902466, -0.5907396674156189, -0.8287640810012817, 1.3366124629974365, 1.4419225454330444, 0.7753289937973022, 0.839493989944458, -0.5790131092071533, -1.1500569581985474, -1.096698522567749, 0.8795410990715027, -0.23293708264827728, -0.5328330397605896, 0.03993185609579086, 0.17077374458312988, -0.2243657261133194, -0.242813378572464, -1.1129876375198364, 0.2718963027000427, 0.2994983196258545, -0.8961912393569946, -0.053918082267045975, 0.48596203327178955, 0.41692012548446655, -0.5641928911209106, -1.1343638896942139, 0.43177247047424316, -0.2698284685611725, -0.43124833703041077, -0.8144455552101135, 0.9864170551300049, 0.7022143006324768, -1.8420953750610352, -1.5448400974273682, -1.8350005149841309, 1.5181549787521362, 1.3355387449264526, -1.8109962940216064, 1.2583614587783813, -0.6831991076469421, -0.015239456668496132, -0.08233977109193802, 1.7222115993499756, -0.21278531849384308, 0.24839739501476288, 0.059799280017614365, -0.3005687892436981, -0.2715238630771637, 1.9294642210006714, -1.0418380498886108, 0.5219760537147522, -0.7099766135215759, 0.374665766954422, -1.554330825805664, -1.9009724855422974, -1.0366166830062866, -0.15755966305732727, 0.6989579200744629, 0.27140092849731445, -0.9747190475463867, 0.9776451587677002, -0.6253715753555298, 1.456701636314392, -0.6882166266441345, -0.7189425826072693, 1.2705421447753906, 0.5633682608604431, -1.253505825996399, 0.5360593795776367, -0.895173966884613, 1.0408645868301392, -0.15392331779003143, -0.9659332036972046, 0.3539333641529083, 1.0141009092330933, 0.28149673342704773, -0.7724047899246216, -0.6713994741439819, -0.24826568365097046, -0.056150615215301514, -1.4752860069274902, -1.2450790405273438, -0.2575012147426605, -0.11400412023067474, 0.19950108230113983, 0.6127384305000305, -0.9791771173477173, 0.969409704208374, -0.2412731796503067, -0.6063093543052673, -0.8400685787200928, 1.3120200634002686, 0.28705060482025146, 2.814682960510254, 0.16433197259902954, 0.5390353798866272, 1.2238091230392456, 1.3532899618148804, 0.029005588963627815, -0.49983686208724976, -0.2961426079273224, 0.4796917140483856, -0.9570403099060059, 0.8441950678825378, 0.04584541916847229, -1.20731520652771, 0.5544565320014954, -1.0110408067703247, -0.6089603900909424, 1.5936423540115356, -1.512790322303772, 1.4767391681671143, -0.5601834058761597, -0.795533299446106, 0.7285310626029968, -0.48137328028678894, -0.38054513931274414, -0.33420485258102417, -2.070796012878418, -0.49174097180366516, -1.0307466983795166, -0.5075471997261047, 1.2273527383804321, 0.33374401926994324, 1.6307058334350586, 2.680147409439087, -1.6761715412139893, 0.5032902359962463, 0.09070664644241333, -1.2764582633972168, -0.18874794244766235, 0.8791295289993286, -1.4708811044692993, 1.246279001235962, -0.3810054361820221, 1.1735715866088867, -0.30992573499679565, 0.3299192786216736, 0.5457744598388672, 0.14569580554962158, -1.5504060983657837, 1.0650818347930908, 1.118560791015625, -0.3689713776111603, 0.23374229669570923, 0.2970750033855438, -1.7261818647384644, -0.3109601140022278, -1.5529978275299072, 0.3970673084259033, -2.2256808280944824, -0.8594540357589722, 1.8163642883300781, -0.266531765460968, 3.407032012939453, 0.491741418838501, 0.9173271656036377, 0.11434440314769745, -0.8167228698730469, 0.31849515438079834, -0.6573961973190308, -1.2343107461929321, -0.1624254733324051, 0.661100447177887, 0.35322442650794983, -1.385669469833374, -0.3346613943576813, 0.3554515242576599, -0.6035338044166565, -0.968271017074585, 0.41301247477531433, -0.037541791796684265, 1.6746143102645874, -0.7063344120979309, -1.7568562030792236, -0.27490514516830444, 0.9900878667831421, 1.1241830587387085, 0.28746911883354187, -0.6953915953636169, 0.7098715901374817, -1.560505747795105, -0.2963481843471527, -0.9183149337768555, -0.2044016718864441, -0.8845607042312622, -1.0336445569992065, 0.3509489893913269, 0.8604379296302795, 0.35276928544044495, 0.59568190574646, -0.9991536140441895, -0.40551531314849854, -0.8569786548614502, 0.7638242244720459, -0.28839778900146484, 1.3065707683563232, 1.8561254739761353, -0.35001489520072937, -0.9181807041168213, -0.41655823588371277, -0.5557668209075928, 0.416570246219635, -0.7503461241722107, 0.3435938060283661, 2.300368547439575, 0.06704066693782806, -0.5762659907341003, -1.330098271369934, -0.8842765092849731, 1.14136803150177, 1.1615877151489258, 0.8195537328720093, 0.3176247477531433, 1.2186113595962524, 0.027969470247626305, -1.3979381322860718, 0.05494971573352814, 0.3677240312099457, 0.5150818228721619, -0.6194475293159485, -0.8277260065078735, -0.6249963045120239, -0.817688524723053, -0.40955835580825806, 1.722495436668396, 0.17115579545497894, 0.4270751476287842, 1.163214087486267, -0.40228861570358276, 1.3804796934127808, -0.3564947247505188, 0.6455888748168945, 0.7273736596107483, -1.604034185409546, 0.40279635787010193, 1.2003921270370483, -0.14733806252479553, -0.33190077543258667, 0.9692484140396118, 1.352946400642395, 2.2460336685180664, -0.02451123669743538, 0.04698396101593971, 0.21320661902427673, 0.26701846718788147, 1.386680006980896, 0.5090867280960083, -0.6540581583976746, -0.546595573425293, 0.5822333693504333, -2.191129207611084, -0.7948417067527771, -0.022802574560046196, -1.219910979270935, -1.336580753326416, -0.9416111707687378, 0.20087572932243347, -1.2468979358673096, 0.35262542963027954, -0.43062734603881836, -1.6181718111038208, -0.45856043696403503, -0.46543586254119873, 0.1392829865217209, -0.6296321749687195, -1.694718360900879, 0.3295922577381134, -1.440684199333191, -0.6981483101844788, 0.6558299660682678, 0.4242893159389496, 0.6130648255348206, 0.46383097767829895, 1.0980671644210815, 0.500045120716095, 1.2210115194320679, 1.5850393772125244, 0.07177432626485825, 0.3565021753311157, 0.6284050345420837, -1.0435914993286133, -1.3779957294464111, -0.36068934202194214, 0.3997417688369751, 0.7976877093315125, -0.8898218870162964, 0.1944141387939453, 1.6115672588348389, -2.614511251449585, -0.6208615303039551, 0.31193017959594727, -0.6776835322380066, -0.6662346124649048, 1.3257108926773071, 1.2703814506530762, 0.5321476459503174, -0.1006803810596466, -0.4059281051158905, -0.2622835636138916, -1.475679636001587, -0.9994933605194092, -0.08272445201873779, -0.5764405131340027, 1.1950879096984863, 0.6420267820358276, 0.9508251547813416, -0.9907151460647583, -0.737204909324646, 1.224191427230835, -1.994075059890747, -1.706114649772644, -0.0047763800248503685, 1.0684456825256348, 2.391746759414673, 0.5045669078826904, -0.46668532490730286, 0.31745001673698425, -2.1054599285125732, -0.3835073709487915, 2.7739241123199463, 2.433513879776001, -0.3740788400173187, 0.799180269241333, 0.7221249341964722, -0.49720484018325806, 2.09248423576355, 1.9153823852539062, -1.7973194122314453, -1.6098130941390991, -0.2537185549736023, 0.08334194123744965, -0.5135772228240967, 0.13391081988811493, -0.601115882396698, -0.24396362900733948, -0.6520140767097473, -0.22015716135501862, -1.1400024890899658, -0.23189781606197357, -1.1940810680389404, 1.7695519924163818, -0.8588255047798157, -0.32882019877433777, 0.42596185207366943, -0.20228615403175354, 0.7272763848304749, 1.3313449621200562, -0.7582661509513855, -0.017092889174818993, -0.5754126310348511, 1.2836377620697021, -0.29542022943496704, 1.2205561399459839, 2.1650469303131104, 0.3474680185317993, -1.2663649320602417, -1.319253921508789, -1.9286075830459595, -1.6582978963851929, -0.839164674282074, 1.5212180614471436, 0.8640387654304504, -0.1763129085302353, -0.9929275512695312, 0.429199755191803, -0.9282464981079102, -0.5077524781227112, -1.3526560068130493, -0.07601366937160492, 1.0832399129867554, -0.30057114362716675, -0.2488265335559845, -1.146825909614563, 1.7182598114013672, 0.31912291049957275, 1.5713361501693726, 0.08439868688583374, -0.2531915605068207, -0.943752110004425, 0.6353136897087097, 1.2942856550216675, 0.21655353903770447, 0.11475615203380585, -2.2672715187072754, -1.2064317464828491, -0.8881377577781677, 1.3025879859924316, -0.04768306389451027, -0.055155396461486816, 0.2919626235961914, -0.10381925851106644, -0.42842474579811096, 1.692508578300476, 0.7695257663726807, -0.8707619309425354, 1.0041602849960327, 0.5779848098754883, -0.1915270984172821, -2.02461838722229, -1.3675514459609985, 1.3182379007339478, 0.0684901773929596, 1.6973029375076294, -0.3472971022129059, -0.6468787789344788, -0.9699841141700745, 0.2155262678861618, -0.8354082703590393, -0.6495449542999268, 1.054193139076233, 1.0435583591461182, -0.5254839658737183, 1.9031559228897095, 0.28904294967651367, -0.7373663187026978, 0.5063063502311707, 1.0147464275360107, -0.3033886253833771, -1.0906513929367065, -0.0324920117855072, -2.5566747188568115, -1.1072707176208496, 0.5863811373710632, 1.075742244720459, -0.45258986949920654, -0.30882659554481506, -0.13709162175655365, 0.995011568069458, -0.2864501476287842, -0.9429592490196228, -0.958831250667572, 0.4429897665977478, 0.6926260590553284, -0.4555387496948242, 0.34394335746765137, -1.4214955568313599, -0.10583312064409256, -0.20196714997291565, 1.5591318607330322, 0.9524824619293213, -0.5818649530410767, 1.1468678712844849, 0.6011127233505249, -0.21911334991455078, -0.02356451191008091, 0.7992019653320312, -0.1888643503189087, -0.17909599840641022, 1.4086945056915283, -0.4280186593532562, -1.1519081592559814, 1.5636721849441528, -0.5086933374404907, -1.4896215200424194, -0.44297555088996887, 0.45400944352149963, 1.1899452209472656, 0.5723981857299805, -0.5958226323127747, 0.25033366680145264, -1.4938583374023438, -1.7787375450134277, -0.6518544554710388, -1.2763237953186035, -0.13751563429832458, 1.6859917640686035, 0.4570184350013733, -0.8207690715789795, 2.3067173957824707, -1.8560423851013184, 1.4258826971054077, 0.17879028618335724, -0.311062216758728, -0.18975190818309784, -1.0854246616363525, 0.052128471434116364, -0.8564174175262451, -1.1871775388717651, -0.6522404551506042, -1.7017121315002441, -1.8105612993240356, -0.908125102519989, -1.0180662870407104, 1.708663821220398, 1.2067333459854126, -0.1683237999677658, 0.25200536847114563, 1.2262455224990845, -0.559711217880249, 0.49607527256011963, 1.168839931488037, 0.776589035987854, 0.9477483630180359, -0.45373106002807617, -1.3852131366729736, 0.5374628305435181, 0.1457233428955078, -1.0332083702087402, 0.9524098038673401, -0.14556072652339935, 0.14467859268188477, 0.2567158639431, -0.254763662815094, -1.2051477432250977, 2.6671974658966064, -1.468798279762268, 0.9179749488830566, -0.8942960500717163, -0.658746600151062, 0.6003727316856384, 0.1655305027961731, 1.1864416599273682, -0.32718968391418457, 0.3298603296279907, -1.2215243577957153, -0.19584354758262634, 0.2942460775375366, -0.7577239274978638, 1.360860824584961, -1.3623062372207642, 0.9238973259925842, 0.7440844774246216, -1.2188003063201904, 0.8256931900978088, -0.6988901495933533, -0.22771106660366058, 0.6064475178718567, 0.8985158801078796, -0.995721697807312, -0.42698904871940613, -1.0507081747055054, 0.1796492636203766, -0.487163245677948, -1.3807435035705566, -0.39301788806915283, 1.8634517192840576, 0.08047746121883392, 1.740573525428772, -0.3285175561904907, 1.1714067459106445, -0.11503259092569351, 1.2636215686798096, 1.2696696519851685, -0.45738813281059265, -1.3530957698822021, -1.1650103330612183, 1.570391297340393, -0.43343356251716614, -1.1292352676391602, -0.06640297174453735, 0.7871195673942566, -1.5277732610702515, 0.714059054851532, 0.5157471895217896, 1.267682433128357, -1.154118537902832, -0.8667867183685303, -0.19650261104106903, 0.3981500267982483, 0.7220058441162109, -0.3879460096359253, -2.2980434894561768, 1.5161415338516235, -0.5779309868812561, -0.7554227709770203, -0.054114989936351776, -1.6838724613189697, -0.18136079609394073, -0.3439442813396454, 0.38147422671318054, 0.292060911655426, 0.8219947218894958, 0.9386425018310547, -0.6261172890663147, -0.23732708394527435, -0.5091294646263123, -1.421362280845642, -0.30263751745224, -1.412842035293579, 1.4554094076156616, -1.452054738998413, -0.027273647487163544, -1.608237862586975, -0.545029878616333, -1.648119330406189, -1.64278244972229, -0.3001171052455902, 0.8822982311248779, -1.9616575241088867, 2.1922709941864014, 0.8799047470092773, -0.2678709328174591, -0.2863948941230774, 0.1720321625471115, 0.04169681668281555, 0.704459547996521, 1.9619324207305908, 0.08304543793201447, 0.8139817118644714, -0.7488183975219727, -1.2835313081741333, 0.40953245759010315, -0.5522298812866211, -0.14497874677181244, -0.4615267515182495, 1.9843381643295288, 1.9054677486419678, -0.7867475152015686, 0.3092259466648102, -2.4444451332092285, 0.8956642150878906, -0.24741438031196594, -0.28226813673973083, -0.5456352233886719, -0.08199760317802429, 1.1095302104949951, -1.2186390161514282, -0.12551257014274597, 0.7543001174926758, -1.065468668937683, -2.6192455291748047, -1.7758196592330933, 0.21656066179275513, -0.3626483380794525, 0.5765145421028137, -0.350757896900177, -0.881662905216217, 0.4302366077899933, 0.6407305002212524, 1.706020474433899, 1.4597840309143066, -0.31051602959632874, -0.48129749298095703, 1.1971181631088257, -0.2864739000797272, 0.48613440990448, -0.21735182404518127, -1.7312906980514526, -1.587373971939087, 0.5765772461891174, -0.1821914166212082, -1.2926387786865234, 0.5892602205276489, -0.34292614459991455, -0.5531240701675415, 0.9094939827919006, -0.12920333445072174, -0.9338998794555664, -0.7517257332801819, -2.29376220703125, -1.3922964334487915, 1.176727533340454, -0.6550858616828918, -0.34649014472961426, -1.6438283920288086, 3.745321750640869, 1.6246039867401123, 0.04705014079809189, 1.4237055778503418, 0.5675758123397827, 0.8871232271194458, 0.9507164359092712, 1.0398503541946411, -1.2738921642303467, 1.6523468494415283, -0.6746087074279785, 1.006351351737976, -0.133157879114151, -1.6962283849716187, 1.8440918922424316, -0.6241013407707214, -0.6389712691307068, 0.5152926445007324, -0.14129114151000977, -1.1543281078338623, -1.0305874347686768, 1.2997914552688599, 1.2880595922470093, -0.13499468564987183, -0.06857915222644806, -0.009724111296236515, -0.04606173560023308, 0.3671870529651642, -1.1469193696975708, 0.3014339506626129, -0.6406339406967163, 0.8580278754234314, 2.405768632888794, 0.4146600663661957, -0.1433040201663971, -1.427119493484497, -1.1737134456634521, 0.13187342882156372, -0.5296167731285095, -0.6652135252952576, 0.19900856912136078, -1.0979048013687134, -1.5221196413040161, 1.3705192804336548, -1.6315085887908936, 0.171031191945076, 0.5231618881225586, 0.05949508398771286, -0.7216838598251343, -0.9913350343704224, -2.2100698947906494, -0.22183391451835632, -1.7367082834243774, -0.29907506704330444, 0.33435356616973877, 1.2489262819290161, -0.02348463237285614, 0.7629013657569885, -0.9533057808876038, 1.310916543006897, 0.2156076431274414, -1.8315911293029785, 0.1839311718940735, 0.6593755483627319, -1.1765687465667725, 0.14537766575813293, -0.6921185851097107, -2.6006078720092773, -0.5277853012084961, 0.27111202478408813, -0.43091848492622375, -0.30402475595474243, 1.1067187786102295, -0.17928697168827057, 0.7586950063705444, -0.06732397526502609, 0.11794818192720413, -0.8416114449501038, 1.591741919517517, -0.5487989187240601, 0.16328361630439758, 0.5086356401443481, 0.5736022591590881, 0.9365057349205017, -0.5474584102630615, 0.49700212478637695, -0.911821722984314, 0.37902307510375977, 1.331886887550354, 0.06271359324455261, -0.7097205519676208, 2.278703212738037, 0.603244423866272, 0.0940030887722969, 1.096213698387146, 0.4982486963272095, -0.4329807162284851, -1.3210043907165527, 0.7566165328025818, -1.4374884366989136, -1.388159990310669, 0.5020832419395447, -1.5351340770721436, -0.5743541717529297, 0.8776223659515381, -0.02733159437775612, 0.2565324902534485, 0.6223948001861572, -0.28271934390068054, -2.5976569652557373, -0.6895646452903748, -0.2171049267053604, 0.5241273045539856, 1.3674067258834839, -2.1458942890167236, 0.3448864817619324, -1.4827858209609985, 0.2858123481273651, -2.0032787322998047, -0.29321327805519104, 0.39821380376815796, -2.3881595134735107, 0.8145549893379211, 1.043161392211914, -1.204369068145752, 0.6571313738822937, 0.01690891943871975, -0.12073589861392975, 0.2960379123687744, -1.1914575099945068, -2.280637741088867, 0.35315796732902527, 1.395477294921875, 1.8526630401611328, -0.4438847601413727, 1.8066929578781128, 0.7424694299697876, 0.567989706993103, -1.1080708503723145, -0.7185845971107483, -0.6041719317436218, -1.2226765155792236, -2.043667793273926, 0.9674786925315857, -0.7265700697898865, -1.0047800540924072, -1.4996448755264282, -0.2921823263168335, -0.14856432378292084, 1.1989895105361938, -0.6207578778266907, -0.3113132119178772, 1.6881903409957886, -0.860665500164032, 1.1930330991744995, 0.9283691048622131, -0.44963538646698, -0.12307471036911011, 2.292053461074829, -0.34064939618110657, 0.26206958293914795, 0.9647694826126099, 0.1400449275970459, 0.6928563714027405, 1.3857651948928833, -1.129812240600586, 0.9636833667755127, 1.0100175142288208, -0.27903077006340027, -0.30598148703575134, 0.11563888192176819, 1.3925706148147583, 0.6444138884544373, 1.0071629285812378, 0.055123768746852875, -0.03091306798160076, -0.7691592574119568, 0.6253261566162109, -0.5200608372688293, 1.1912914514541626, -0.4387335479259491, -1.642578125, 0.6190859079360962, 0.009745581075549126, 0.00045854825293645263, -0.032158028334379196, -0.8252912163734436, 1.201310396194458, -0.6602078676223755, -0.9794839024543762, 0.11528832465410233, 1.1723525524139404, -0.4483190178871155, 0.32828089594841003, 0.9615364074707031, -0.068479984998703, -1.1182544231414795, -0.4305819571018219, -0.9467443823814392, 0.2667335569858551, -0.8359549045562744, -0.007853293791413307, -0.11716913431882858, 1.4889116287231445, -0.45454105734825134, -0.5776592493057251, -2.3655049800872803, -1.730132818222046, -1.0399200916290283, -1.7986241579055786, 1.2060943841934204, 1.6422433853149414, -1.1174840927124023, -0.42287757992744446, 0.5828990340232849, 0.12228243798017502, -1.557578682899475, 0.04793795198202133, -0.7170249223709106, -0.8594350814819336, -1.3649680614471436, -1.124224066734314, -1.7252010107040405, -0.7138075232505798, 0.14402072131633759, -0.053388334810733795, 2.779775619506836, -0.7582330703735352, -0.36297717690467834, 0.0199457798153162, 0.21936920285224915, 0.6401368975639343, 0.8350799083709717, 0.5310549736022949, 0.9064376950263977, -1.8662583827972412, -0.2865161597728729, -0.273388147354126, 1.4991494417190552, 1.0952916145324707, 0.37333106994628906, 1.4409388303756714, -1.1859008073806763, -0.941829264163971, -0.26312732696533203, -0.5337027311325073, -0.38870126008987427, -1.5945249795913696, 0.1699477881193161, -1.6614989042282104, -0.20027868449687958, -0.5201136469841003, -0.012775260955095291, -1.0102297067642212, 0.5568591952323914, 1.888318419456482, -1.132386565208435, 0.09175083786249161, -0.5847876667976379, 1.7565290927886963, 0.5139594078063965, 1.0435863733291626, 0.7859266400337219, 1.2710579633712769, -0.4511927664279938, 0.27189722657203674, 0.20331662893295288, 0.5592160224914551, 1.3556920289993286, 2.055842161178589, 0.614621102809906, 1.3490254878997803, -1.3434653282165527, 0.13500456511974335, -1.2340978384017944, 0.04981112852692604, 2.3623223304748535, 0.5196090936660767, -0.17617081105709076, 1.9906773567199707, 1.0164601802825928, -0.4899001121520996, -0.009422951377928257, 0.5023535490036011, 1.2292033433914185, 0.3618464469909668, 0.8230948448181152, 0.3924836814403534, -0.7243006229400635, -1.7291560173034668, 0.19580945372581482, 0.11621985584497452, -1.9105257987976074, -0.4608144462108612, -1.6261191368103027, 0.5884192585945129, 0.2391841858625412, -0.9224536418914795, -1.5073094367980957, 1.0249401330947876, -1.5522736310958862, -0.4538564383983612, -0.5183737277984619, 1.6523323059082031, 0.20794795453548431, -0.48834261298179626, -0.3591756224632263, -0.9125630855560303, -0.4627021253108978, 0.011490635573863983, 1.512888789176941, 0.313517689704895, -1.1421358585357666, -1.8430463075637817, 0.8050400018692017, -0.4132554233074188, 0.6978748440742493, 0.26279428601264954, -0.7207984328269958, 0.5738524198532104, 1.2411932945251465, -0.5557411313056946, -0.7619161009788513, -1.5241847038269043, -0.5888078808784485, 1.040704607963562, -0.5837973952293396, 0.4951063394546509, 0.43746718764305115, 0.7244243621826172, -0.7608020305633545, -0.5007438063621521, -0.49293437600135803, -2.744713544845581, -0.08971123397350311, 0.142746701836586, -0.5207922458648682, -0.1180943101644516, 1.3971407413482666, 0.600574254989624, 0.32518473267555237, 1.61701238155365, 0.9487293362617493, -0.39330267906188965, 0.3271876871585846, 0.6227611303329468, 0.2578417658805847, 1.010118007659912, 0.9924941062927246, -0.8443643450737, -2.1802477836608887, 0.6955316662788391, -0.10095157474279404, -1.2051007747650146, 0.9093818664550781, -0.2054361253976822, 0.08160177618265152, 0.51493239402771, 0.9381426572799683, 0.23165263235569, 0.33014723658561707, -0.793660581111908, -0.22621391713619232, 0.2473164051771164, -0.1167345717549324, -8.92388925421983e-06, -0.11021897196769714, 0.9009385704994202, 0.8672188520431519, 1.1966142654418945, -1.0236808061599731, 0.557807981967926, 1.1447941064834595, 0.16990973055362701, 0.3914007246494293, 1.0149946212768555, 2.0155081748962402, 1.835533618927002, 0.16140581667423248, -2.0891432762145996, -0.9419994950294495, 0.6136380434036255, -0.7088102698326111, -0.6563783884048462, -1.0450739860534668, -0.4479433298110962, -0.6895504593849182, 0.0038226826582103968, 0.9220422506332397, 1.2787601947784424, -0.5560193657875061, -0.7224224805831909, 0.2818105220794678, -0.6693255305290222, 0.8225820660591125, 0.26819777488708496, -1.0756560564041138, 0.2773960530757904, 0.5388432145118713, -0.719999372959137, 0.67875075340271, 1.0583150386810303, -0.4310680627822876, -0.4887799024581909, -0.3146856725215912, -0.9821486473083496, 1.5783312320709229, -0.7826228141784668, -0.530070424079895, 0.706935465335846, -0.7454975843429565, 0.6436124444007874, -0.8415666818618774, -0.8993290066719055, -0.3652465045452118, 0.7590745091438293, 0.0008487994782626629, -0.8601704835891724, -0.7386524081230164, 1.4356573820114136, -0.8329395651817322, 2.0747063159942627, 0.3946056067943573, -0.3387094736099243, -0.5117741227149963, -1.4144535064697266, -0.12299822270870209, 0.8364388942718506, 1.310837984085083, 0.14149032533168793, 1.4274070262908936, 0.7766431570053101, -1.0586764812469482, 0.6681346893310547, -0.44814640283584595, 0.32283616065979004, 1.0076698064804077, 0.6330604553222656, -0.08427481353282928, 0.9513357877731323, 2.3218815326690674, -0.004549954552203417, -1.0019378662109375, -0.959427535533905, 0.6764768958091736, 0.7570951581001282, 1.3931690454483032, 0.3331315815448761, 0.5662073493003845, -0.8489203453063965, 0.9933055639266968, -0.11168079078197479, -0.7471067309379578, 1.3454920053482056, -2.084324836730957, 0.5047416687011719, -0.6961972713470459, -0.7390182018280029, -0.8121689558029175, -2.766979217529297, 1.5188384056091309, 0.2533189356327057, -0.3574758470058441, -1.0932066440582275, 0.05281902104616165, 0.6872179508209229, -0.9509150981903076, 0.3692080080509186, -1.9083302021026611, 1.5151103734970093, 1.4375795125961304, -1.4930601119995117, 2.275905132293701, 0.08395946770906448, 1.061921238899231, 1.396279215812683, 0.5840280055999756, -0.05561710521578789, -0.9114383459091187, 0.3868967592716217, -0.6632123589515686, 0.3465147018432617, 0.14948654174804688, -0.31814441084861755, 1.095873475074768, -3.1628856658935547, 0.6336776614189148, -1.4833425283432007, -2.446693181991577, 0.843666672706604, -0.379066526889801, 0.23513920605182648, -2.1514241695404053, 0.32187777757644653, -0.7305595874786377, 0.7376793026924133, 0.6701954007148743, -0.7478466033935547, -2.5218966007232666, -1.4181475639343262, -1.7122905254364014, -0.5669829845428467, -1.170350432395935, 0.959273636341095, -0.3624055087566376, 0.5610866546630859, -1.8974990844726562, -0.49859505891799927, -1.5216166973114014, 2.3200929164886475, -1.2600198984146118, 0.12856994569301605, 0.15675565600395203, -0.4769212007522583, -0.18364179134368896, 0.397431880235672, 1.4010637998580933, 1.1700208187103271, -1.0230387449264526, -1.0713335275650024, 0.37595099210739136, 1.4902275800704956, 0.5999279022216797, -1.4858243465423584, -0.3509925603866577, -0.5909597873687744, -0.6820223331451416, -0.2303057610988617, -1.1609808206558228, -1.4651976823806763, -0.8902321457862854, -0.375201016664505, -0.20352166891098022, 0.6246955990791321, -1.6885123252868652, 0.2193712592124939, -1.7772034406661987, 1.7726497650146484, -0.7224498987197876, -0.7018769383430481, 1.5531126260757446, 0.29663875699043274, 1.5946818590164185, -0.45451125502586365, 0.24009884893894196, 0.6425747871398926, 0.004599447827786207, 0.6202489137649536, -0.7891594171524048, -1.634664535522461, -1.2868833541870117, -0.4839639663696289, -0.05086202546954155, 1.2484768629074097, -0.7670984864234924, -1.4183530807495117, 0.34544193744659424, 0.23587436974048615, -0.21929992735385895, 1.2499490976333618, 1.3733676671981812, -0.27244144678115845, -0.5251258015632629, -0.29082390666007996, 1.086897373199463, 0.07580583542585373, 0.15880930423736572, -0.16377922892570496, 1.1097846031188965, 0.24928699433803558, 0.7314479351043701, -0.9692187309265137, 1.676601767539978, 1.204617977142334, 0.38549375534057617, -0.8355698585510254, -1.3717578649520874, 1.7387092113494873, 0.3102034330368042, -0.6901766061782837, 0.39645177125930786, -0.7550874948501587, 0.657943069934845, 1.3108772039413452, 0.6000210642814636, 0.8724038004875183, 0.4921768605709076, 0.7227873206138611, -0.7386078238487244, 0.04608594998717308, -0.08313693106174469, -0.3150545656681061, 0.17356964945793152, -0.11938587576150894, 0.5042118430137634, -0.4475418031215668, 1.4977610111236572, -1.1772221326828003, 0.44392409920692444, -0.544768214225769, -0.440594881772995, 0.07879211008548737, -2.3251149654388428, 0.7008022665977478, 1.4706624746322632, 0.5878085494041443, -0.9024770259857178, 0.2649261951446533, 0.6308509111404419, 0.5910199880599976, -0.5334627032279968, -0.10981239378452301, 0.7858771681785583, 0.7247535586357117, 0.5450833439826965, 0.3314500153064728, -0.1379861682653427, 1.742216944694519, -0.4530715048313141, -0.3657234311103821, -0.06934645771980286, 0.21000954508781433, -0.6021422147750854, 0.8757098913192749, 0.31792113184928894, 0.5394952893257141, -1.5859814882278442, -0.6280501484870911, -0.08452462404966354, -0.11097253859043121, 1.2566049098968506, -1.0425574779510498, -0.765830934047699, 0.8357403874397278, -0.5589271187782288, -0.8348608016967773, -0.4465908408164978, -0.0003990585100837052, -1.283316969871521, 0.4671805500984192, 1.1685681343078613, 0.257199227809906, 2.229053258895874, 0.19891364872455597, -1.3488103151321411, -1.1907048225402832, 0.6958604454994202, 0.4525561034679413, -2.399615526199341, -0.1822439730167389, 0.2665397822856903, -0.41735589504241943, 0.012971808202564716, -0.8017515540122986, 0.12080968916416168, 0.38464823365211487, -1.3329999446868896, 1.0553888082504272, -0.013703173026442528, -0.2912599444389343, 0.47077658772468567, 0.6888812780380249, 1.9199509620666504, 0.47038155794143677, -0.518882691860199, -1.0579763650894165, 1.2071642875671387, -0.04070718586444855, -2.8917346000671387, 0.016120489686727524, -0.8943570256233215, -1.9955040216445923, -0.08664434403181076, 0.9510160088539124, -1.7296000719070435, 0.08267446607351303, 0.49640798568725586, 0.546162486076355, -0.06903987377882004, -0.6391805410385132, 0.5116983652114868, 1.542442798614502, -0.02004564180970192, -0.7960397005081177, 0.8142598867416382, 1.373187780380249, 0.30470630526542664, 0.9685282111167908, 0.24380850791931152, -0.07637707889080048, -1.724348545074463, -0.6662653684616089, -0.5107810497283936, 2.8538591861724854, -0.993309736251831, -2.3725523948669434, -0.2977212071418762, -0.8329981565475464, -0.018095409497618675, -0.5569674968719482, -0.109711192548275, -0.9951866865158081, -2.92414927482605, 2.3324697017669678, -0.16046525537967682, -0.3058682084083557, -0.11060898005962372, 0.08169103413820267, 0.29337573051452637, -0.6987193822860718, -0.19879429042339325, -0.08790719509124756, 0.10595511645078659, -0.4472138583660126, -0.1435822695493698, 1.144677996635437, -1.7284194231033325, -0.3922290503978729, 1.372506022453308, -0.964333713054657, 0.08624286949634552, 1.009398341178894, 1.014416217803955, 1.008925437927246, -1.6124309301376343, 1.1685243844985962, -0.8738570213317871, 1.162124514579773, -0.2560581564903259, -0.954964280128479, -0.886915922164917, -0.26983556151390076, 0.9048919081687927, -0.8092146515846252, 0.7342617511749268, -0.9497584104537964, -0.8030257821083069, 0.625942587852478, -1.0943557024002075, 0.1810000091791153, 1.2850524187088013, 0.12337823212146759, 0.2841653823852539, 0.0214301235973835, -0.2711569368839264, 0.7886168360710144, 0.9024351835250854, -0.6812788844108582, 0.02426045760512352, 0.260917067527771, -0.1801312416791916, 0.028417903929948807, -1.2670807838439941, -0.18720455467700958, -2.6121530532836914, -1.630263090133667, -0.20031113922595978, -1.0717356204986572, 0.6178013682365417, -0.2902403771877289, -0.10641979426145554, -0.48699280619621277, 0.30742210149765015, 0.04815037548542023, 0.2137661725282669, -0.45090731978416443, -1.6422771215438843, -0.5656088590621948, -0.8622700572013855, -1.2421483993530273, -1.354980707168579, 2.082348346710205, 1.2631498575210571, 1.8314735889434814, -0.7514576315879822, -1.4317216873168945, -0.3810124099254608, -0.2968777120113373, 0.5927522778511047, 0.40136218070983887, -0.026449700817465782, -0.5477684140205383, 1.3597843647003174, -1.0277001857757568, -0.7616027593612671, 1.0281821489334106, 1.5777690410614014, -0.03385232761502266, 0.11416381597518921, 0.20129959285259247, 1.0923757553100586, 0.8235674500465393, -1.5015894174575806, 0.5669746398925781, 0.38527512550354004, 0.06393612921237946, -1.0409120321273804, -1.0154796838760376, -1.1884669065475464, -0.021350493654608727, 0.1337123066186905, 1.7071788311004639, 1.1332029104232788, -0.1484721153974533, 0.7760878801345825, 0.8737978339195251, -1.9362921714782715, -0.4790833294391632, 0.5464501976966858, 0.024519355967640877, 0.2123783379793167, -1.847615122795105, 0.5101431608200073, -0.08361730724573135, -0.6771215200424194, 0.16144520044326782, -0.029162827879190445, -0.5692773461341858, 0.35213324427604675, 0.9313347339630127, 0.4420584738254547, 0.8686100244522095, 0.5703476667404175, -0.7745404243469238, -1.511821985244751, -1.01270592212677, -1.0853941440582275, 0.4343976378440857, -0.9472789764404297, -1.4018739461898804, 1.0555593967437744, -0.3936845362186432, 0.6834412217140198, -1.0375869274139404, 0.2570347785949707, -0.32561102509498596, 0.9174550175666809, 1.306199550628662, -1.0040839910507202, -0.23527002334594727, -0.05288359895348549, 2.512751340866089, -0.0407751090824604, 1.4993094205856323, 1.8574904203414917, -1.8058953285217285, 1.8452180624008179, 2.9173567295074463, -0.6245312690734863, 1.704518437385559, 1.0782169103622437, 1.0182279348373413, -0.1942724883556366, -0.31679388880729675, -0.20789247751235962, -0.38265523314476013, -0.44215112924575806, 0.662211000919342, -0.40040358901023865, -0.717819094657898, 0.9842612147331238, -1.1454883813858032, -0.5390437245368958, -0.36915794014930725, -0.1888578236103058, -1.5585554838180542, 0.1439896672964096, 0.42399877309799194, 0.7037056684494019, -0.31324538588523865, -0.5299341082572937, -0.37750670313835144, 0.4858338534832001, -1.0485987663269043, -0.526360273361206, 0.6578375697135925, 0.3062449097633362, 0.37556949257850647, 0.1428717076778412, -0.8916618824005127, -1.1205254793167114, 0.2523374855518341, -0.24564547836780548, -1.556182861328125, 1.3731876611709595, -0.7743118405342102, 1.4894551038742065, -0.358355313539505, 0.5053087472915649, 0.053768500685691833, -0.12843255698680878, 0.4269810914993286, 0.9077964425086975, -0.6739730834960938, 0.133095383644104, 0.04642954841256142, -0.15402965247631073, -0.3427886664867401, -0.6183571219444275, -1.9051481485366821, -0.17001411318778992, 0.6564283967018127, 0.323075532913208, 1.308441162109375, 0.5744759440422058, -0.5999817252159119, -0.9779791831970215, 1.1300581693649292, -1.019239068031311, -1.4246110916137695, 0.17924928665161133, -0.08435007929801941, 0.316348135471344, -0.8941284418106079, -1.0226320028305054, -0.4885447323322296, 1.5623723268508911, -0.39620858430862427, -0.9144463539123535, 0.6031738519668579, 0.6978588700294495, 0.2889181673526764, -2.5126185417175293, -0.4221273362636566, -0.929129421710968, -0.23459778726100922, 0.18879273533821106, 0.7493801116943359, 0.2013426423072815, 2.412498712539673, -1.2712422609329224, 1.254353404045105, 0.888658881187439, -1.0594514608383179, 0.9749894738197327, -1.0398188829421997, 1.4727778434753418, -0.35118982195854187, -0.6378804445266724, -1.9039185047149658, 0.5241590142250061, 2.722681999206543, -0.04084239527583122, 1.2378461360931396, 0.9993348717689514, 0.5597293972969055, 0.9534846544265747, -0.022187627851963043, -0.14489462971687317, -1.2020423412322998, 0.16007894277572632, -1.277746319770813, 0.3490195870399475, 1.0660905838012695, -0.45911431312561035, -0.7109293341636658, 0.21537554264068604, -0.4584546387195587, 2.074502468109131, -1.2112120389938354, -0.5570091009140015, 0.9491868615150452, 0.42661309242248535, -0.7448325157165527, 0.7143552899360657, 0.8301839232444763, 0.8175298571586609, -0.14983852207660675, 1.1409528255462646, 2.1677396297454834, 1.6263923645019531, -2.4244587421417236, -0.2561735510826111, -0.06228506565093994, -0.4951767921447754, -1.0979820489883423, 0.3610006868839264, 0.07305934280157089, 1.0409647226333618, -0.8234391212463379, -0.698589026927948, 0.21663516759872437, 1.0657771825790405, -0.01827380433678627, 0.00508621521294117, -0.7841835021972656, 0.27555567026138306, 1.2823177576065063, -0.8320940732955933, -0.4383428990840912, -0.634369969367981, -2.607402801513672, -0.4395599663257599, -0.8206948041915894, -1.0162252187728882, -0.6980000734329224, 0.5546241998672485, 1.5134429931640625, -0.1964692324399948, -0.7014961242675781, -1.6425588130950928, 0.16456100344657898, 0.6583896279335022, 0.007964764721691608, 1.2312811613082886, 0.5176624059677124, -1.6098520755767822, 0.5075491666793823, -1.6669484376907349, -1.7618227005004883, 0.3262809216976166, 0.27640169858932495, 0.765123188495636, -1.8660873174667358, 0.36766767501831055, 1.0964910984039307, -2.118999719619751, 0.1363319605588913, -0.5879469513893127, -2.2988173961639404, 2.706954002380371, -1.127652645111084, 2.2192463874816895, -0.19739949703216553, -0.16133122146129608, -0.30423322319984436, -0.3475123643875122, 0.11814361810684204, 0.51966792345047, 1.1606054306030273, -1.012046217918396, -0.7525752186775208, -1.2197344303131104, 2.151423215866089, 0.09018825739622116, 0.4124625027179718, -0.4477556645870209, 1.5904146432876587, 0.4379778504371643, -1.1517834663391113, -0.4507831633090973, -0.6376240849494934, -0.569163978099823, 1.5353355407714844, 0.3478814363479614, 0.18995650112628937, 2.142671823501587, -1.5054296255111694, -0.7163518071174622, 0.14689014852046967, -0.708371102809906, -0.27605003118515015, 0.6011077165603638, -2.753300189971924, -1.3872677087783813, 0.7838711142539978, 1.036520004272461, -1.0093121528625488, 1.7880808115005493, 0.6592634916305542, -1.9274779558181763, -0.0337233692407608, 0.6955491900444031, -0.12251982092857361, 1.9533683061599731, -1.4708486795425415, -0.7151502966880798, -0.6984291672706604, 0.20694392919540405, 0.9290273785591125, -0.05022641271352768, -0.9029095768928528, 1.0062683820724487, 0.09158320724964142, 0.13396131992340088, -0.3957267701625824, -0.1696111112833023, -0.5158966183662415, -1.3918896913528442, -0.37152206897735596, 0.0965709462761879, -0.0258182343095541, 1.564720630645752, -1.0918517112731934, 0.6389519572257996, -0.40039587020874023, -1.5762691497802734, -0.18006661534309387, 0.31644195318222046, -1.9678950309753418, 1.8324800729751587, 1.1550090312957764, 0.09234900027513504, 0.7993406653404236, 0.48376160860061646, 0.7965168952941895, 0.1225385069847107, 0.7241362929344177, 1.10287344455719, 0.9860833883285522, 0.7251824736595154, 0.6782048344612122, 0.12119074910879135, -0.6703208684921265, 0.5495101809501648, -1.562978982925415, -1.62684166431427, 0.0198384840041399, -0.9430288076400757, -1.8846923112869263, 2.277132272720337, 0.595159113407135, 2.012551784515381, 1.7819346189498901, -1.9648537635803223, 0.5788702368736267, 0.5706920027732849, -0.9230180382728577, -0.24029739201068878, 0.876807689666748, 0.012886830605566502, -0.3393253684043884, 0.9170387983322144, -0.06265556812286377, 0.6295704245567322, 0.9183470606803894, 0.9318159818649292, -0.021291591227054596, -1.232191801071167, 0.4573989808559418, 0.14138969779014587, 0.43990060687065125, -1.14273202419281, 2.500786542892456, 1.686188817024231, -1.036493182182312, -0.061904843896627426, 1.7390227317810059, -0.6459177732467651, -0.36648887395858765, -1.4762639999389648, 1.9984381198883057, 0.06740807741880417, -2.587796926498413, -2.371143341064453, -0.7314316034317017, -1.573167085647583, -0.9916627407073975, -2.9427103996276855, 0.8009358048439026, -0.32014524936676025, 0.5044265389442444, -0.8347041606903076, 0.5049395561218262, -1.3881514072418213, 0.28535300493240356, -1.1511049270629883, 1.261014461517334, -0.35944512486457825, -1.2125030755996704, -0.028488673269748688, 0.26186805963516235, 0.5003910660743713, -0.294564425945282, -1.2303314208984375, 1.3567497730255127, -0.4542754292488098, 1.759842872619629, 0.21982558071613312, -0.17926451563835144, -0.9575747847557068, 0.9721818566322327, 0.7306556105613708, 1.0145885944366455, -1.0009069442749023, 0.28769344091415405, -0.12227138876914978, 0.05991772562265396, 1.4767247438430786, -0.8341080546379089, 0.416262686252594, 1.4375767707824707, -0.3250752091407776, 0.15980254113674164, 1.6068484783172607, -0.8021062612533569, -1.6508983373641968, 1.2503725290298462, 0.8333117961883545, 0.34981560707092285, 0.5626449584960938, -1.194563627243042, -1.0398412942886353, -0.27001526951789856, -0.09410981833934784, 0.2390277236700058, -0.07988452166318893, 1.2902666330337524, -1.7227567434310913, -0.7092609405517578, 0.3978433907032013, 0.8940630555152893, -1.8252604007720947, -1.1758372783660889, 1.0627185106277466, -0.18486839532852173, -1.1954340934753418, 0.4961372911930084, -0.1616179347038269, -1.6675105094909668, -1.1839357614517212, 0.7491784691810608, -0.46299678087234497, -0.1861078292131424, 1.1525204181671143, 0.12723812460899353, 0.38679468631744385, -0.4692379832267761, -0.1873801201581955, 2.426785945892334, -1.4980252981185913, 1.3828551769256592, -1.9515291452407837, -0.686586856842041, -0.656097948551178, -1.9026135206222534, -0.5556638836860657, 1.2719993591308594, 1.3927732706069946, 0.3742271959781647, 0.10115427523851395, -0.8688594698905945, -0.3290502429008484, 1.9810882806777954, -1.7533622980117798, -1.1809847354888916, -0.21856330335140228, 0.30456599593162537, 0.3370477855205536, 0.08213803172111511, -0.3396880328655243, 0.7141208052635193, 0.39621907472610474, 1.073603630065918, -0.8374894857406616, 0.842925488948822, -1.146264672279358, -0.5601543188095093, 0.05106436461210251, -0.14395438134670258, 0.02234250120818615, 0.4088231921195984, -0.27649614214897156, -0.5736373066902161, -0.39193668961524963, -0.636952817440033, 0.4651128053665161, -1.3892301321029663, -0.2030927538871765, 0.6038344502449036, -0.0633854791522026, 0.9005662202835083, -0.08808882534503937, 1.558703064918518, -0.00774837052449584, -0.8535533547401428, -0.633358359336853, 1.9029479026794434, -0.2947463095188141, 0.8939740657806396, -1.95054292678833, -0.3337472081184387, 0.2144584208726883, -0.022571047767996788, 1.1547369956970215, -1.473030924797058, 0.09589261561632156, -0.032677870243787766, -0.8613468408584595, 1.050857424736023, -0.606708288192749, 1.2435628175735474, -0.9915785789489746, 2.178621530532837, -0.3561711013317108, -1.7538764476776123, 1.1821837425231934, 1.0191560983657837, -0.6282672882080078, -0.06078290194272995, 0.30551445484161377, -2.1191775798797607, 0.7993548512458801, 0.6293895244598389, -0.01205353531986475, 0.18990416824817657, 0.2298213243484497, 0.387073814868927, -0.24816212058067322, 0.06484393030405045, -0.6792907118797302, -1.1196273565292358, 1.1345880031585693, 0.14385922253131866, -0.6584994196891785, -0.8642117977142334, 0.9504316449165344, 1.7231593132019043, -0.15645015239715576, 0.24420832097530365, 0.7362900376319885, 0.1330084502696991, 0.04743127524852753, -1.1340303421020508, -0.3671371638774872, -0.8983705043792725, -1.5920319557189941, -0.5141633749008179, -1.1626734733581543, -0.6318753361701965, -1.5905609130859375, 0.24609288573265076, -0.24280714988708496, 0.39244794845581055, 0.7133004665374756, -0.45549148321151733, 0.8566965460777283, -0.18129144608974457, -0.2368122637271881, -0.8963731527328491, -0.4700359106063843, 0.6602639555931091, 0.4925936758518219, 0.8393124938011169, -0.23874859511852264, 0.26629549264907837, 0.1967991590499878, -0.36029767990112305, -1.7551153898239136, -0.8136860728263855, -0.8768463134765625, -0.38480427861213684, -1.0365571975708008, 0.2985953390598297, 0.20845256745815277, -0.020101958885788918, -0.7282689213752747, 0.15114980936050415, 1.1295989751815796, -0.003442991292104125, 0.8610147833824158, -0.6211716532707214, -0.6205811500549316, -0.07069581001996994, -1.0585192441940308, 2.283412456512451, -1.1960779428482056, -0.6773201823234558, -0.4493696391582489, 0.07095373421907425, -0.8323245048522949, 0.1558123677968979, -0.7077999711036682, -0.4266985058784485, -0.9068167209625244, 1.1685805320739746, -0.9325326681137085, -0.3428797125816345, 0.8604890704154968, 0.4741351306438446, 0.43327295780181885, -1.7704498767852783, -0.28893834352493286, -1.07381010055542, -0.05007396638393402, -1.0072576999664307, -1.0912861824035645, 0.5087207555770874, -0.936847448348999, -1.1749500036239624, 0.5914541482925415, 0.392681747674942, -0.633962094783783, 0.03536557778716087, -0.1515941470861435, 0.25502756237983704, 0.24209441244602203, 1.542428731918335, 0.7616281509399414, 0.7449573278427124, 0.2957316040992737, -0.6881312727928162, 0.24299375712871552, -0.8292950391769409, -0.5632538199424744, -0.06407327204942703, -0.7208942770957947, -1.0094130039215088, -0.8422556519508362, -0.3029918968677521, 1.7913068532943726, -0.1266450583934784, -0.635034441947937, -1.3651930093765259, -0.2326192408800125, -0.410338431596756, 0.31580910086631775, -0.9011124968528748, -1.9323862791061401, 0.7532129883766174, -0.33745554089546204, -2.1923828125, -1.6858421564102173, -1.2477803230285645, 0.18979966640472412, -0.673322856426239, 0.30940988659858704, 0.2077772617340088, 1.5678213834762573, 1.183876395225525, -2.6507568359375, -0.6366657018661499, 0.6349110007286072, -0.20903518795967102, -1.490135669708252, 0.4461641311645508, 0.22610004246234894, -0.8176918625831604, -0.5723651051521301, -0.5470530390739441, 0.19971349835395813, -0.13194067776203156, -1.1065139770507812, 1.2508347034454346, -1.089569091796875, 1.5297365188598633, 0.6581241488456726, 0.7293963432312012, -0.6339771747589111, 0.2735293507575989, 0.5892791748046875, -1.3894131183624268, -0.8482689261436462, -0.7003480195999146, 0.23900778591632843, -0.07764484733343124, -0.7535355091094971, -0.5513397455215454, 0.8933568596839905, -1.1006826162338257, -2.3180460929870605, -1.084567904472351, 0.12259852141141891, -0.2718811333179474, -0.013171227648854256, 0.3289671242237091, -0.5106034874916077, 1.0232300758361816, 0.4687756299972534, 0.8995862603187561, 1.2588307857513428, 1.7106997966766357, -1.1502434015274048, -1.039361596107483, 0.47516459226608276, 0.011474192142486572, -0.968808650970459, 1.1728594303131104, 0.36565685272216797, -0.7943130731582642, -0.4940258860588074, -0.8635442852973938, -1.6430821418762207, -0.1957998275756836, -1.158211588859558, 0.6652297377586365, -0.15256302058696747, -0.0849972814321518, 1.2857255935668945, 0.8856328129768372, 0.18924716114997864, 0.2844406068325043, 1.0709539651870728, -0.010677937418222427, 1.498780608177185, 0.2616746425628662, -0.670791745185852, 0.4730249345302582, 0.8221149444580078, -1.7533695697784424, -0.5918440222740173, -0.6624870300292969, -0.13402169942855835, 0.47582364082336426, -0.5545372366905212, 0.931751012802124, 0.03279579058289528, -1.9530872106552124, 0.3022691607475281, -0.9161322116851807, -0.244651198387146, -1.0724835395812988, -0.9688366055488586, -2.762284517288208, -1.2049256563186646, 0.7585212588310242, -1.0547322034835815, 1.0923963785171509, -0.8778428435325623, -0.682424783706665, 0.7755764722824097, 0.30812907218933105, -0.05743173509836197, -0.9695345163345337, 0.5205239653587341, -0.768275260925293, -1.2324066162109375, 0.20907506346702576, -0.11822869628667831, 1.802626132965088, 0.44674935936927795, -0.09512914717197418, -1.344911813735962, -2.2969088554382324, 0.3832729756832123, -0.4188549518585205, 1.8125962018966675, 0.28559157252311707, 0.4559638500213623, 1.7660883665084839, -1.3188493251800537, -0.6073253750801086, 1.3326565027236938, -2.231778860092163, 0.3859202265739441, -0.2873852849006653, -0.35197126865386963, 1.420077919960022, -1.1062601804733276, -0.338840514421463, 0.7922833561897278, 1.0427615642547607, 0.6809905171394348, 1.0003314018249512, 0.5875503420829773, -1.0097724199295044, 0.021484512835741043, -3.0616302490234375, 1.5431928634643555, 0.12225208431482315, -0.5849440097808838, -2.129671573638916, 0.2731655538082123, -1.312322735786438, 0.28302767872810364, 0.06025470420718193, -0.9060571789741516, 3.261932134628296, 0.6127657294273376, -0.6087132096290588, 0.38267555832862854, 1.994999647140503, 0.30846819281578064, 0.5046063661575317, -1.4715250730514526, -0.0842890590429306, -0.26200562715530396, -1.0135525465011597, -0.610036313533783, -0.1413974165916443, 0.46733716130256653, 0.07382534444332123, 0.16325050592422485, 0.530443549156189, -0.5213367342948914, -1.025771141052246, -0.9643287062644958, 0.4981570541858673, 1.0089274644851685, -0.6178786754608154, -0.27616897225379944, -1.907623052597046, -1.2032766342163086, 0.24234165251255035, 0.7879814505577087, 0.10598186403512955, -2.1838648319244385, -1.6507781744003296, 1.3014166355133057, 0.05232350528240204, -1.225017786026001, -0.33493587374687195, 1.029137134552002, 0.8690451979637146, -0.5076148509979248, -0.2516511380672455, -1.0338046550750732, 0.43736395239830017, 1.0791021585464478, 0.26385900378227234, -0.2518874704837799, -0.7938507199287415, -1.0870380401611328, 0.24191516637802124, 0.433001309633255, -1.1097509860992432, -0.8246138095855713, -1.11826753616333, 0.27646854519844055, -0.1138434186577797, -0.13443595170974731, -0.4836794435977936, 1.189333200454712, 0.06535331904888153, -0.7510536313056946, -0.5024067759513855, 1.7862799167633057, -0.3188484013080597, -0.27629342675209045, -2.1444427967071533, -1.0313771963119507, 1.3005691766738892, -0.41370174288749695, 2.8504602909088135, 0.0020361733622848988, -0.40687426924705505, -1.3078126907348633, 1.0141394138336182, 2.4983789920806885, 0.573920726776123, -0.29618385434150696, -0.3738339841365814, -0.9556670784950256, 0.2671715319156647, 0.5908428430557251, -0.26442933082580566, -1.0316680669784546, 0.8556583523750305, -0.771146833896637, 0.5811911225318909, 0.8871150612831116, -1.4034686088562012, -0.6765422821044922, 0.8996162414550781, -1.9734218120574951, 0.4062472879886627, 0.14223140478134155, -0.11931432783603668, 1.5328924655914307, -0.6437938213348389, -1.0261982679367065, -0.7700517177581787, -1.3681052923202515, 1.8109784126281738, 1.46890389919281, -1.845445156097412, 1.4051450490951538, -0.337025910615921, -2.765695333480835, 1.9395854473114014, -0.018790777772665024, -0.5788857936859131, -0.04674769937992096, -0.03099212795495987, 0.8835248351097107, -0.38366127014160156, 0.9887665510177612, 0.2723526060581207, 0.05732429400086403, -1.3015276193618774, 0.33894044160842896, 0.8094867467880249, -0.2500886619091034, 0.015721092000603676, -0.6812711358070374, -0.906150221824646, 0.1697208285331726, 0.7707070708274841, -0.01828492246568203, 0.22210894525051117, -0.5665714144706726, 1.3130865097045898, -1.4630382061004639, 0.3973677456378937, 3.166018009185791, -1.3602195978164673, -0.5334148406982422, -1.6887972354888916, 0.2186732143163681, -0.8367043137550354, -0.2619244456291199, 0.7173000574111938, -0.3415483236312866, 0.36942368745803833, 1.1906516551971436, -0.4160378575325012, 0.06636041402816772, 1.1378229856491089, -0.9149020910263062, -0.05426817387342453, 1.0148460865020752, -0.3380339443683624, 0.07814033329486847, -0.7484519481658936, 0.3379864990711212, -0.16420145332813263, 0.4457922577857971, 1.5608258247375488, -1.5800600051879883, -0.6709778308868408, 0.9098095893859863, -1.4365283250808716, 0.9306061863899231, -0.4551573395729065, 1.3698475360870361, -1.0273914337158203, -0.769778311252594, 0.4179445505142212, -1.0851022005081177, 0.12703613936901093, -1.2511742115020752, 0.7112282514572144, 0.04425116628408432, -0.26560911536216736, -0.40856844186782837, 0.34265515208244324, -0.3283507525920868, 1.678329348564148, -0.5011346340179443, 0.9988299012184143, 1.0820167064666748, -0.9307529926300049, 0.6147346496582031, 0.751287579536438, 1.1897344589233398, -0.8405042886734009, 0.7440164685249329, -0.8814767003059387, 0.8835269808769226, -0.05780927836894989, -0.34582704305648804, 0.18663813173770905, -0.883079469203949, -1.606689691543579, -1.4488219022750854, 0.8295746445655823, -0.7119810581207275, 0.2901473939418793, -1.2013198137283325, 0.6170828342437744, 0.5104566812515259, 0.8789532780647278, -0.11439432203769684, 1.6358646154403687, -1.144874095916748, 0.3985670506954193, 0.8501705527305603, -2.1813273429870605, 1.75977623462677, 1.8217315673828125, 0.3897070586681366, 0.7117267847061157, 0.5799792408943176, -1.150826096534729, -0.015491275116801262, -0.11778225749731064, 0.7503884434700012, -1.4049952030181885, -0.01954324170947075, -1.0525250434875488, -1.9204894304275513, -0.8475840091705322, -0.9619325399398804, -0.22109274566173553, 0.23348933458328247, 0.3626232147216797, 0.0021517507266253233, 1.0263571739196777, 0.3280273377895355, 1.2242355346679688, 0.6442118883132935, -0.3045079708099365, 0.9975399971008301, -0.48901402950286865, -0.4498119354248047, -1.280687928199768, 1.0111429691314697, -1.729256272315979, -0.39201024174690247, -0.6851708292961121, 1.1089690923690796, 0.28149858117103577, 0.8410994410514832, 0.030379287898540497, -0.4307454228401184, 0.2788013815879822, 0.6227519512176514, 0.3637065589427948, -0.30080732703208923, -1.0497981309890747, 0.07884060591459274, 0.12948188185691833, -0.5419088006019592, 0.7231758236885071, -0.089645154774189, 0.8155712485313416, -0.2191651314496994, -0.4659826457500458, 1.5973920822143555, 0.9167194962501526, -0.07511359453201294, 0.3824978768825531, 0.05621188133955002, 0.4273033142089844, -0.09519992768764496, -1.9907139539718628, -0.12626785039901733, -1.336328387260437, 0.2335822731256485, -0.9072120785713196, -0.15012206137180328, 0.44301068782806396, -0.09555727243423462, -1.7227846384048462, -0.9527963995933533, -0.4040401875972748, 0.08820437639951706, 0.13089780509471893, -1.844027042388916, -0.4280032813549042, 2.1735594272613525, 0.36418384313583374, 0.6639798283576965, -0.39476144313812256, 2.017503499984741, -0.18842676281929016, -0.5055148601531982, -0.3486942946910858, 0.11537439376115799, 0.6238740682601929, -0.8841874003410339, -0.43672114610671997, 1.5828142166137695, -0.7354905009269714, -1.0360910892486572, -1.308127999305725, -0.31986233592033386, -0.6292370557785034, -0.10963577032089233, -0.6071353554725647, 0.5987547039985657, -0.8340529799461365, 0.04608974978327751, -1.02017343044281, 1.5623921155929565, -0.4903062880039215, 0.3431450128555298, -2.163411855697632, -1.35873544216156, 0.032603226602077484, 0.8367090225219727, -0.020646052435040474, 1.7421543598175049, 0.4803929030895233, 0.5679348707199097, -1.523469090461731, 1.6960986852645874, -1.2599244117736816, 1.2090991735458374, -0.3715408742427826, 0.3446844816207886, 1.024896502494812, 0.4454745650291443, -0.6483328938484192, -0.3967646360397339, -0.2538841664791107, 0.32588645815849304, 1.5929021835327148, 1.1798537969589233, 1.3597742319107056, 2.0055460929870605, -0.2671448886394501, -0.1268358826637268, -2.1142637729644775, -2.628139019012451, -0.7607326507568359, -0.004572608973830938, -0.5268606543540955, -0.6106559038162231, 0.42891907691955566, 0.2955162525177002, 1.1561391353607178, -1.314822793006897, 1.0057384967803955, -0.18331876397132874, 0.29146143794059753, 0.41298213601112366, -0.3826134204864502, 1.1514606475830078, -1.0315898656845093, 1.0447224378585815, 0.8222315311431885, -0.5742719769477844, -1.2538477182388306, 1.4634957313537598, 0.605612576007843, -0.44502779841423035, -1.437085509300232, 0.13243158161640167, -0.7999559044837952, -0.2706575393676758, 0.8203942179679871, 0.25997477769851685, -0.44154784083366394, -0.20029228925704956, 0.8294737935066223, -0.8829835653305054, 1.4901846647262573, -0.2752838432788849, -0.6609764695167542, 1.8801887035369873, -0.15656855702400208, 1.2605397701263428, -3.3455097675323486, -0.5746932029724121, -0.1512540727853775, -0.8428502082824707, 1.2567509412765503, -0.06659805029630661, -0.5681791305541992, -0.6340416669845581, -0.7120065093040466, -0.06782092899084091, 0.07033982127904892, 0.72788405418396, -0.10190552473068237, -1.509574055671692, 1.0489072799682617, -0.4471619427204132, 0.48351266980171204, -1.0646471977233887, 0.7747107148170471, 1.043075442314148, -0.42941051721572876, -0.0654674768447876, 0.10029575228691101, 0.9359205365180969, -0.8656647205352783, -1.7336804866790771, -0.8177365064620972, 1.337380051612854, 0.0353424996137619, -0.9034327864646912, 1.591583251953125, 0.18044275045394897, 0.5403084754943848, -1.062842845916748, -0.6462830901145935, 2.6194798946380615, -0.7338448166847229, 0.9000707864761353, 0.3227940499782562, -0.8259139060974121, -0.06857544183731079, 0.5529322028160095, -0.5883851051330566, 0.39375820755958557, -0.5169348120689392, 0.9147698283195496, -0.7906599044799805, 1.2805787324905396, -0.2855094075202942, 1.060707688331604, 0.20272964239120483, 0.16400083899497986, -0.9020416140556335, 1.2447086572647095, 0.18922032415866852, -0.9919777512550354, -0.03017299994826317, -1.2287887334823608, 1.0263588428497314, -0.22974438965320587, 0.3889206349849701, -0.5955235958099365, -1.1837233304977417, 0.39820724725723267, 0.6568886041641235, 2.0810387134552, 2.2470483779907227, 0.1079002097249031, 1.2732734680175781, 0.24275656044483185, -1.1970784664154053, -0.17348654568195343, -2.0728867053985596, -1.0847196578979492, 1.0020469427108765, -0.7647110819816589, -0.7388530969619751, 2.5131189823150635, 0.26451677083969116, -0.3639286458492279, 0.8551701903343201, -2.23274302482605, 0.37256282567977905, -0.003931631799787283, 0.8053514361381531, -0.1619035005569458, 0.605697751045227, -0.011829429306089878, -1.9904710054397583, 2.079080104827881, -0.9295618534088135, 0.649267315864563, 0.0014447380090132356, 1.341937780380249, 1.8955121040344238, 0.5858696103096008, -1.8369568586349487, 0.3310677707195282, -0.5961076617240906, -1.6107966899871826, -0.7017862200737, -0.11960631608963013, -1.1131641864776611, -0.7402961254119873, -0.3890305757522583, 0.029100241139531136, 1.1524906158447266, -0.5044669508934021, -1.4721540212631226, -0.2934376299381256, 1.6308788061141968, 0.3460749387741089, -0.9836718440055847, -2.238377571105957, -0.2777095437049866, 0.13683778047561646, -1.1003317832946777, -0.5660496354103088, 0.6392690539360046, 0.5353356003761292, 1.7590398788452148, -0.7403224110603333, -0.9429947137832642, 0.14669841527938843, 1.2739360332489014, -0.40639621019363403, -1.2355124950408936, 2.0631539821624756, 0.6362378001213074, -0.44698938727378845, -1.4380604028701782, -0.46121397614479065, 0.9162693023681641, -0.02519293874502182, 1.0517367124557495, -0.033095818012952805, 0.6185285449028015, 1.545863151550293, -0.35158342123031616, 0.058320123702287674, 0.8265409469604492, 1.720687747001648, -0.7633669972419739, -0.7574745416641235, 0.4175901710987091, 0.8749694228172302, 0.4698058068752289, -0.4447360634803772, -0.20285028219223022, 0.6578208208084106, 0.7115339040756226, -2.2775259017944336, 0.035911496728658676, 0.6222533583641052, -0.6013767123222351, 0.6782625913619995, 1.1754281520843506, 0.7328943610191345, -0.4500531554222107, 0.6287411451339722, -0.8199462294578552, -0.19850309193134308, 1.1156524419784546, -0.8784544467926025, 0.9293211698532104, -0.10273386538028717, -1.3943464756011963, -2.419874429702759, 0.5043627023696899, -0.36494553089141846, 2.452157497406006, -1.2288191318511963, 0.6776426434516907, -0.6122294664382935, 0.6221920251846313, 1.336069941520691, -0.3416486084461212, -0.5304020643234253, 1.0903724431991577, -0.9923776984214783, 0.859652042388916, 0.4241613447666168, 1.3479219675064087, 0.018217764794826508, -0.21458566188812256, 1.0306016206741333, -1.7944667339324951, 0.06477369368076324, 0.40418562293052673, 0.9068313241004944, -1.3590633869171143, -2.5990161895751953, 0.44947245717048645, 0.3616989254951477, -2.502368927001953, 0.8253366947174072, -1.0237189531326294, -0.37020596861839294, -0.5444979071617126, -1.4921964406967163, 1.7143847942352295, 0.9197587966918945, -0.6856968998908997, -0.042267221957445145, 1.2420005798339844, 0.8628625273704529, -2.1471245288848877, -0.041352931410074234, -1.0983489751815796, 0.0361650250852108, -1.1303948163986206, 1.0367169380187988, -1.3383331298828125, -0.3012882173061371, -0.734001100063324, -1.7868131399154663, -0.4120195806026459, 1.5066947937011719, -0.26096096634864807, 0.2226775586605072, 0.1403854936361313, -1.172926664352417, 0.6542859077453613, -0.7645173668861389, -0.7479131817817688, -1.669140338897705, 1.8095163106918335, 0.21040964126586914, 0.46867015957832336, 1.411434292793274, 1.3956680297851562, 0.3235837519168854, -0.7657883167266846, 0.46639493107795715, -0.7505356669425964, -0.43624642491340637, -0.2393118143081665, 0.1065932884812355, -0.3549690544605255, -1.9273102283477783, -1.4079874753952026, 0.7170854806900024, 0.1781775802373886, 1.749624490737915, 0.861547589302063, 0.9290803670883179, -0.4888300895690918, 0.6311473250389099, 0.8613989353179932, 0.09060721099376678, -2.036856174468994, 0.4811325967311859, 1.397628664970398, 0.21522755920886993, 0.2500399053096771, -0.07583411782979965, 1.0561898946762085, -0.24044685065746307, -0.2949793338775635, 0.3531983196735382, -0.6484454274177551, 0.31784260272979736, -0.09417475014925003, -0.048614051192998886, -1.2833136320114136, -0.7394731640815735, 1.0759263038635254, -0.16589829325675964, 0.6248306632041931, 0.551196277141571, -0.35641977190971375, -0.6516821384429932, -0.7672023177146912, -0.9010663032531738, -0.877693235874176, -0.6844055652618408, -0.25172311067581177, 1.1925773620605469, 0.27068209648132324, 0.06258847564458847, 0.990466296672821, 0.5414049625396729, 0.018300604075193405, 0.47542664408683777, -0.7134308815002441, -0.32835492491722107, 0.5542263388633728, -0.5630671977996826, 0.7705563902854919, -0.07089220732450485, 0.4087222218513489, -1.63430917263031, 0.05661759525537491, -0.7681028842926025, -0.5239861607551575, 1.2502403259277344, 0.7428559064865112, 0.2793187201023102, 0.27796244621276855, -0.8499684929847717, 0.38229432702064514, 1.577326774597168, -0.11067933589220047, 1.6270813941955566, 0.4913630783557892, -0.7048245668411255, -0.4829203188419342, 0.29378455877304077, -0.7905619144439697, -0.6188622713088989, -0.06711166352033615, -1.3275961875915527, -0.012093337252736092, 0.08268158882856369, -0.6422615051269531, -0.16316743195056915, -1.677432894706726, -1.0481367111206055, 1.3139878511428833, 1.4009002447128296, 0.8475983738899231, 1.2761863470077515, 0.37131351232528687, -0.8438331484794617, 1.2800774574279785, 0.44755202531814575, 0.4532633125782013, -0.301790326833725, 2.0359582901000977, -1.280746340751648, 1.04482102394104, -0.37984466552734375, -0.2846653461456299, -0.09003475308418274, 0.054458096623420715, -0.7636213302612305, -1.065962553024292, -0.0590822733938694, 0.44166675209999084, 1.3545279502868652, -0.704512357711792, -0.06926488876342773, 0.6281595230102539, 2.1682538986206055, -0.2597520649433136, 0.9474695920944214, 1.3754963874816895, -0.913996160030365, -0.7643733620643616, 0.10030064731836319, -1.9051532745361328, 0.11502130329608917, -1.980686902999878, -1.529944658279419, -0.49680766463279724, -0.24641847610473633, -0.07581962645053864, -0.08831536769866943, -0.676155149936676, -0.516007125377655, 0.17403331398963928, -0.9370130300521851, 0.6810781955718994, -0.31081199645996094, 0.984635591506958, -0.026425154879689217, -2.0336005687713623, -0.8471055030822754, 0.09832317382097244, -0.2763656675815582, 1.2136765718460083, 2.3031554222106934, 1.8790812492370605, 0.6135134100914001, -0.1905437558889389, 0.7204197645187378, -0.8317966461181641, -0.2938927412033081, -0.5021458864212036, 1.41361403465271, -0.2897825837135315, -0.5713961124420166, -0.30705228447914124, 0.7767612934112549, 0.19719365239143372, 0.5165910124778748, -0.47756120562553406, 0.5849567651748657, 1.7159132957458496, 1.4171168804168701, 1.3103227615356445, -0.2639743983745575, -0.2985296845436096, 0.7887337803840637, 1.133322834968567, 1.3305788040161133, -1.023848295211792, -1.4487355947494507, -0.8865958452224731, 0.897000253200531, 1.1804221868515015, -0.14519943296909332, -1.6372264623641968, -0.33097609877586365, 0.4094306230545044, 1.2013901472091675, -0.8695579171180725, 1.7106142044067383, 0.44395148754119873, -1.1941534280776978, 2.2958083152770996, -0.20283111929893494, -1.9161651134490967, -1.8243156671524048, -0.38286930322647095, -0.35115280747413635, 1.1483738422393799, 1.72151517868042, 0.656090497970581, -0.11960635334253311, -0.2333836704492569, 1.030756950378418, -1.2461248636245728, -0.13055330514907837, -0.5751543641090393, 0.346740186214447, -0.2045954018831253, -0.5847928524017334, -0.09083796292543411, 0.5682426691055298, -1.2117438316345215, 2.123220443725586, -0.5526574850082397, 1.0003087520599365, 0.12868919968605042, 0.6276085376739502, 1.6666122674942017, -0.4130569398403168, 0.8310394883155823, -1.6354552507400513, -1.28266179561615, 1.295167088508606, -0.8244718909263611, -1.4583030939102173, 0.39221590757369995, -1.1758402585983276, -1.2627452611923218, 0.24584133923053741, -0.7328211665153503, -0.2365655153989792, -1.033259630203247, -0.4607863426208496, -1.1587060689926147, 2.205483913421631, 1.4106152057647705, 0.08602327853441238, -1.2359331846237183, 0.2648506760597229, -0.16052091121673584, 0.3579767346382141, -0.6059461832046509, 0.7679584622383118, 1.3879202604293823, -0.47016045451164246, -0.31366658210754395, -1.8855719566345215, 1.4685291051864624, 1.3867532014846802, -0.2962312698364258, -1.1287063360214233, -0.6117852926254272, -0.18158121407032013, -1.2970694303512573, 1.3304272890090942, 1.219207525253296, 0.27112898230552673, 0.8816474676132202, -1.0609159469604492, -1.5044108629226685, -0.5624956488609314, -0.007391427643597126, -1.1846591234207153, -0.9841324687004089, 0.28855499625205994, 1.1549732685089111, 6.009820572216995e-05, -0.8425053954124451, -1.7205004692077637, -1.481573462486267, -0.19148112833499908, -1.5838344097137451, -0.9184209108352661, 0.588030993938446, 0.45188868045806885, -0.5855823159217834, 1.8466753959655762, -0.5239156484603882, -1.1393266916275024, -0.42882758378982544, -0.2153429090976715, -1.4185885190963745, -0.2651911973953247, -1.224010705947876, 0.0418851375579834, -0.14272600412368774, -0.9587569236755371, 0.13869650661945343, -0.5386208295822144, 1.0334495306015015, 0.7194175720214844, 0.32767173647880554, 1.2053396701812744, -1.083141565322876, -0.7661884427070618, -1.9476083517074585, -0.13471010327339172, -0.6168257594108582, -0.7131804823875427, 2.0586202144622803, 0.2246047705411911, 1.291505217552185, 0.45612239837646484, 0.3523319959640503, 0.6131556630134583, 0.8102995753288269, 1.4383864402770996, 0.7867556214332581, 0.5517703294754028, -1.315292477607727, -0.31542786955833435, -0.4172004461288452, -0.700583279132843, -0.3593660295009613, -0.3091953694820404, 0.7436099052429199, -2.9037063121795654, 0.2703130543231964, 0.2890990972518921, 2.4365711212158203, 0.38272473216056824, 3.431685209274292, 1.026242733001709, 1.4795050621032715, 1.029412031173706, -1.3097492456436157, 0.6220697164535522, -1.3843085765838623, -0.39452674984931946, -0.005407434422522783, -2.4865829944610596, -0.4725193679332733, -0.23941072821617126, 0.93793123960495, 0.1576121598482132, 1.475946307182312, 0.2761854827404022, -1.1637494564056396, 0.09552401304244995, -2.1834990978240967, -0.3207208216190338, -1.1790062189102173, 0.8020097613334656, -1.0405303239822388, -0.17073966562747955, 0.7009422183036804, 0.2790582478046417], [-0.20174340903759003, -0.5478796362876892, 0.05503758415579796, -1.3216456174850464, -0.32114121317863464, -0.6306880116462708, 0.5982816219329834, 1.0449233055114746, 1.870458722114563, -0.1324976533651352, 1.386425495147705, 1.0582640171051025, 1.9954580068588257, -0.30781909823417664, 0.2861226201057434, 0.5776888132095337, 0.4575154781341553, 0.8474841117858887, 1.230619192123413, -1.4181160926818848, -1.169273853302002, 0.4704534709453583, 1.049156665802002, 1.0693511962890625, 0.6390025615692139, -1.7986644506454468, 0.7559100389480591, 1.5832653045654297, -0.6819499135017395, 0.16545909643173218, -1.0481376647949219, 0.05370870977640152, 0.004785296972841024, 1.974911093711853, -0.44276270270347595, -1.2367459535598755, -0.9138333201408386, 0.9710506200790405, 0.0019826453644782305, 1.3824102878570557, 0.2598711848258972, 0.8457819819450378, -0.962344765663147, 0.761784017086029, 0.5575467348098755, 1.4708850383758545, 0.2878864109516144, 0.3404247462749481, -0.454005628824234, 1.3779155015945435, 1.0902730226516724, -0.4113187789916992, -2.286729574203491, -0.35433411598205566, 0.4766414761543274, -1.797300100326538, 1.0258351564407349, -0.9795163869857788, -1.3543323278427124, -0.9647665023803711, -0.5509561896324158, 1.1568248271942139, 1.5732351541519165, 0.031529348343610764, -0.9642705917358398, -0.6872888207435608, -0.05579762160778046, 0.04257262125611305, -0.7334988117218018, -0.9526211023330688, -0.40749967098236084, 0.9945976734161377, 0.849779486656189, 0.869483232498169, -0.9130027294158936, 1.6820365190505981, -0.0751265212893486, -0.6271568536758423, 2.0280916690826416, 0.43801093101501465, 0.08293942362070084, 1.8794904947280884, -1.9113669395446777, -1.2626328468322754, -1.296462893486023, 1.5636107921600342, 0.18636898696422577, -0.21780544519424438, -0.8758242130279541, 0.5274128317832947, -0.6163297295570374, -0.47881731390953064, -0.03587321192026138, 0.7043458223342896, -1.1572842597961426, 0.605495810508728, -0.3501090109348297, 0.6383269429206848, 1.7774344682693481, -0.5345205068588257, 0.7860981822013855, -0.43065309524536133, -0.25744837522506714, 0.3298884928226471, 0.6564654111862183, 0.6080690026283264, 1.672275424003601, 1.3376131057739258, 0.6018391847610474, 0.8494634032249451, 0.4087439477443695, -0.6757965683937073, -1.400227427482605, 1.1860547065734863, -0.7686160206794739, -0.47213777899742126, -1.4008266925811768, 0.2211906462907791, 0.6858929991722107, 0.6924996376037598, -2.288856267929077, 0.5709754228591919, 0.38663291931152344, -0.49356701970100403, 1.088489294052124, 0.08836881816387177, 0.4462885856628418, 0.41185319423675537, 0.2037392258644104, 0.4007696807384491, 0.5111433267593384, 0.7991869449615479, 0.5495537519454956, -0.41664919257164, 0.34166309237480164, 0.8062978386878967, -1.0991559028625488, 1.015148639678955, -0.2929271459579468, 0.6995242834091187, 0.5217286944389343, -0.07144664973020554, -0.9445086121559143, 0.17422837018966675, -0.7260168790817261, -0.7631303071975708, -1.0625979900360107, -0.36940720677375793, -1.4380849599838257, 0.19412720203399658, 1.4761186838150024, 1.4073312282562256, -0.16409382224082947, 0.3309459686279297, -1.1100653409957886, 0.42465218901634216, 0.035986755043268204, -0.4516127407550812, -2.491939067840576, 0.17615927755832672, 1.3277150392532349, -0.5136567950248718, 0.2336438149213791, 1.5768160820007324, -0.22724638879299164, -0.10384847968816757, 0.5096296668052673, -0.8162952065467834, 0.14676940441131592, -1.1021281480789185, -2.0450077056884766, -1.3191235065460205, -0.5144399404525757, 0.19510900974273682, -0.3722812831401825, 0.6712175011634827, 0.9509323239326477, -0.28247860074043274, -2.2244300842285156, 0.09220614284276962, -0.32362890243530273, 0.008153153583407402, -0.7766808867454529, 0.7299035787582397, -0.26984474062919617, -1.0014501810073853, 0.07096700370311737, -1.6738895177841187, 0.09124742448329926, 0.7697726488113403, -0.4027540683746338, 1.537468433380127, 0.6352583169937134, -1.211707592010498, -0.10308144986629486, 0.06771384924650192, 0.03046450950205326, -0.40089932084083557, 0.5684332847595215, 1.5479941368103027, 0.4364928901195526, -1.178844690322876, -1.3566257953643799, -0.3729041516780853, 0.6995018720626831, -0.3917500674724579, 0.8138577938079834, 0.06552470475435257, -0.7727251052856445, 0.7572386860847473, 0.39133787155151367, -0.8628445267677307, 0.26413795351982117, -0.7152853608131409, 0.04670990630984306, 0.16347669064998627, 1.2807003259658813, -0.2341366559267044, -0.3905911147594452, 0.660345196723938, 0.2491115927696228, -0.742331862449646, 0.8565248847007751, 0.7364881038665771, -0.4084075689315796, 1.4504942893981934, -0.6844601035118103, -1.294039249420166, 0.500248908996582, -1.4081432819366455, -0.31177768111228943, -0.07228896766901016, -0.9892560243606567, -0.9002042412757874, 1.6933401823043823, -1.1897677183151245, 0.07515861839056015, -0.4827750027179718, -0.017073482275009155, -0.8225582838058472, -0.7857039570808411, 0.8465430736541748, 1.923186182975769, -0.08124710619449615, 1.0362721681594849, -0.7126986384391785, -0.5731884241104126, 1.5537381172180176, 0.05986308306455612, -0.49648016691207886, -1.419801115989685, 0.17548725008964539, 1.6714575290679932, 0.659343421459198, -2.3628525733947754, -2.8064191341400146, -0.3655543327331543, 0.9733834862709045, -0.006663623731583357, -0.27261167764663696, 2.2067203521728516, 0.2800322473049164, 1.0553234815597534, -2.1158273220062256, 0.1737397462129593, -1.8996256589889526, 0.1385800540447235, -1.0381505489349365, 2.076638698577881, -0.42843589186668396, 0.015305127948522568, 0.14556530117988586, 0.11528435349464417, -0.15153682231903076, -0.02483103796839714, -0.13355396687984467, 0.5291063189506531, -2.258730411529541, 1.1331064701080322, 1.7042568922042847, -0.19260576367378235, 0.9687639474868774, -0.650905430316925, -1.4220662117004395, 0.8802136182785034, 1.4263019561767578, 1.5565918684005737, -0.26803332567214966, -0.6308436393737793, -0.15737147629261017, 0.8712020516395569, 0.5415161848068237, 0.09044695645570755, 0.6824815273284912, 0.7951219081878662, -0.7426447868347168, -2.1292660236358643, 0.7504758238792419, 1.0488409996032715, -0.16857178509235382, 0.29174020886421204, -0.6197749972343445, -0.7694383859634399, 0.26708170771598816, 1.3549023866653442, 0.5530240535736084, 1.1856638193130493, -0.2841684818267822, -1.6371619701385498, 0.874151349067688, 0.7874468564987183, -0.545509934425354, 0.8740171194076538, 0.29152849316596985, 0.9204334616661072, -0.4415341317653656, 0.0943044126033783, -0.06708105653524399, -0.4079991281032562, 0.1257271021604538, 0.4230078160762787, -0.5267640352249146, 0.21547666192054749, 2.0119102001190186, -0.5668506026268005, -0.25520023703575134, -1.2131959199905396, -0.9718766212463379, -1.081205129623413, 0.1767793893814087, 0.19816534221172333, 1.738468050956726, 0.8716681003570557, 0.11874859035015106, -1.1381231546401978, 0.9292259812355042, -1.0604326725006104, 0.65312260389328, -0.4758036136627197, 0.7346698045730591, 1.013909935951233, 0.5440104007720947, 0.6702109575271606, 0.07754974812269211, 1.008639931678772, 2.4170191287994385, 0.2624925971031189, -0.0687820240855217, -1.196433186531067, -0.9276489019393921, 0.036718208342790604, 0.4780416190624237, -0.7008717060089111, -0.7550326585769653, -1.6334617137908936, 0.7197625637054443, 0.205710768699646, -0.7881825566291809, 0.4192229211330414, -0.7194717526435852, 0.6480105519294739, -1.657430648803711, -0.4192924201488495, 1.5808308124542236, -1.3194074630737305, -0.34480956196784973, 1.3510559797286987, 0.6377151608467102, -0.03474350646138191, -0.906477153301239, 0.7343661785125732, -1.237531065940857, -0.6939758658409119, -1.4535748958587646, -0.9289900660514832, -1.5094610452651978, 2.3350069522857666, -1.3492504358291626, -1.4380989074707031, -1.011734127998352, 0.19131700694561005, 0.4080571234226227, -0.23697175085544586, 0.1829705387353897, -0.6587638854980469, -0.34200188517570496, 0.8791700005531311, -0.13853992521762848, -1.6209152936935425, -0.6285062432289124, 0.8611820340156555, 1.4428373575210571, -0.08890078961849213, 1.112932562828064, 0.7361260652542114, 0.9849357604980469, -0.6303364038467407, 0.32082897424697876, -1.6135261058807373, -0.2713730037212372, -0.6383771300315857, 0.01794900745153427, 0.08023615926504135, 0.523097038269043, -0.9977846145629883, 0.41153836250305176, -1.4925991296768188, 1.7927188873291016, 0.03756087273359299, 0.6428576707839966, 0.8374186158180237, 1.8845349550247192, -0.7452253699302673, 1.2858229875564575, -1.8675439357757568, 0.4029999375343323, 0.3844916820526123, 0.32686853408813477, -0.533478856086731, -0.7914729714393616, 0.8020597696304321, 0.20535297691822052, -1.0764614343643188, -0.1989385187625885, -0.14545519649982452, 0.9682019352912903, -1.2178343534469604, -0.10952448099851608, -0.5208727121353149, -1.864335298538208, 0.22575004398822784, -0.31942006945610046, 2.1548776626586914, 0.3788972795009613, -0.3085551857948303, 1.132046103477478, -0.5759072303771973, 0.5088741779327393, -1.643903374671936, 0.044888366013765335, 1.787620186805725, 0.3196788728237152, 1.6977894306182861, -0.8415802121162415, 0.9953269958496094, 1.3377526998519897, -1.1466825008392334, -0.24717427790164948, 0.3316247761249542, 0.06187405064702034, -0.729383647441864, -1.47767174243927, 0.4892369210720062, 0.06093661114573479, 0.2377743124961853, -0.4606466293334961, -0.23425157368183136, -2.320629596710205, -2.4555699825286865, -1.0332732200622559, -0.7811789512634277, -1.072307825088501, 0.6800166368484497, -0.2782241702079773, 0.3258654475212097, -0.024162452667951584, 2.2507097721099854, 0.6363120675086975, -0.24842700362205505, -0.04620983824133873, -1.501325011253357, 0.6733028888702393, -0.08570866286754608, -0.8633176684379578, 1.1025868654251099, -0.5520002245903015, 0.3264489471912384, 0.44782793521881104, -1.3219906091690063, -0.8598172068595886, -0.615972101688385, 1.2205233573913574, -0.5131582617759705, 1.2976434230804443, 1.4389539957046509, -0.7844875454902649, -0.40214869379997253, -0.4101368188858032, 0.6704946160316467, -1.0754960775375366, 1.9001258611679077, -0.8045457601547241, -1.018420934677124, 0.6187137961387634, 0.38450565934181213, 0.6476868987083435, 0.2391871213912964, 0.6637760996818542, -0.03656068444252014, -0.010472551919519901, 1.48783540725708, -0.0786597803235054, -1.8658019304275513, 0.34787285327911377, 0.7549391984939575, 0.35544848442077637, 1.0178778171539307, -1.1032636165618896, 0.28489798307418823, -0.1516626477241516, 0.9346482753753662, -0.21260084211826324, 1.2691502571105957, -0.008455386385321617, 1.0292752981185913, -0.7163693904876709, -2.4418582916259766, 1.3910249471664429, 1.0408966541290283, -0.031486138701438904, 1.1554464101791382, -0.9937970638275146, 1.4984999895095825, -1.1606522798538208, 0.0805291160941124, -1.132978916168213, -0.9069877862930298, -0.06585691124200821, 0.4388149380683899, 0.7692159414291382, 0.4135376214981079, 0.6488232612609863, -2.2764980792999268, -0.5708948969841003, -0.45172303915023804, 1.068302869796753, 0.15544985234737396, 0.728009045124054, 1.5021684169769287, -0.4728061258792877, -1.3109915256500244, -0.29282328486442566, -0.06762965023517609, 0.21807746589183807, -0.7595719695091248, -0.7646808624267578, 0.15289174020290375, 0.10827551037073135, -0.021991446614265442, 0.5913573503494263, 0.10051455348730087, 0.18947733938694, -0.34717386960983276, 0.954394519329071, -1.0543930530548096, 1.0014307498931885, -0.39693573117256165, -0.8674289584159851, -0.9773392677307129, 0.8150376677513123, -1.090063452720642, -2.044752836227417, -0.13173776865005493, 0.037421829998493195, 1.1068546772003174, -0.30451974272727966, -1.4885485172271729, 0.2348833531141281, -1.3995060920715332, 0.6640466451644897, -0.43193623423576355, -1.6700633764266968, 1.1452542543411255, -0.873420000076294, -0.6787858009338379, 1.0165736675262451, 0.5973811745643616, -0.9505068063735962, 0.7951701283454895, -1.1742221117019653, -0.5496968626976013, -0.9397101402282715, -2.052610158920288, -1.3881773948669434, 0.06482524424791336, -0.6135287880897522, -0.5391474366188049, -1.7016456127166748, 0.07796105742454529, 0.19270361959934235, 1.8349227905273438, 0.7668830752372742, -0.26456496119499207, 2.0332658290863037, -0.7477953433990479, -0.2006787806749344, -0.34586167335510254, 0.8692784309387207, 0.19518524408340454, 2.2584524154663086, 1.414369821548462, 1.081917405128479, -1.4515020847320557, 0.583063006401062, -0.9460042119026184, -0.4149610996246338, 0.30769988894462585, 0.7124972939491272, 2.4423885345458984, -1.091925024986267, 0.8511945605278015, 0.043385472148656845, -0.3119055926799774, 0.9935191869735718, 0.15436191856861115, 1.4044378995895386, 1.1311849355697632, 1.9444854259490967, 1.438350796699524, -0.9061415195465088, -1.8375391960144043, 0.21344898641109467, -0.5328238010406494, 1.3748717308044434, 1.4499036073684692, 0.73308926820755, -0.5451699495315552, -0.259554922580719, -1.0244851112365723, -0.8087949156761169, -0.17253302037715912, 0.16370294988155365, 0.6766366958618164, 1.320632815361023, 0.5657753348350525, -0.8066163063049316, 0.34308576583862305, -0.8483297824859619, 0.9046429395675659, 0.7824791073799133, -0.17224732041358948, -2.0889697074890137, 1.3159464597702026, -0.5079037547111511, 1.075714111328125, 0.3758612871170044, 0.019761105999350548, 1.0260119438171387, -0.3269110321998596, 0.6968603730201721, 0.30697908997535706, 1.8115943670272827, 1.1500906944274902, 0.32413536310195923, -1.094923496246338, -0.7539809942245483, -0.6429345011711121, 0.633736252784729, -0.6529386043548584, -0.5027405023574829, 0.9429849982261658, 0.22175590693950653, 0.5900689959526062, 0.4253556430339813, -0.08369326591491699, 1.2369306087493896, 0.1939990222454071, 1.4825173616409302, 2.623441696166992, 0.44480907917022705, 0.25466611981391907, 0.7105072140693665, -0.8754259943962097, -0.8750966191291809, -1.3139530420303345, -1.199059247970581, 0.3943878412246704, -1.1822105646133423, -0.17243388295173645, 0.755415141582489, -1.4168901443481445, 1.287110447883606, -1.1095985174179077, 1.5290946960449219, -1.3577455282211304, -0.13366739451885223, -0.9925842881202698, 0.485979825258255, 0.27316808700561523, 0.0927811712026596, -1.1030590534210205, -0.5448612570762634, -1.6658228635787964, 0.03399118408560753, 0.17031915485858917, 0.34059786796569824, -0.05699736624956131, 1.0681672096252441, -0.16941919922828674, -1.2076479196548462, -0.07811042666435242, -1.6693570613861084, 0.4195573031902313, -0.16335979104042053, -0.7929502725601196, -0.8416579365730286, -0.8941947221755981, 0.4036058783531189, -2.8594820499420166, 0.25709426403045654, 0.004724840167909861, -0.16190864145755768, 2.310560703277588, -1.5846681594848633, -1.5480855703353882, -0.005887101870030165, -0.3519977033138275, -0.8570747375488281, 1.2966985702514648, 0.326619029045105, -0.2310055047273636, 1.7939679622650146, -0.7979462146759033, 0.3778408467769623, -0.7362437844276428, 2.323411703109741, -0.6217491626739502, -0.7704861760139465, -1.2821598052978516, 1.5138808488845825, 0.1573498547077179, 0.6219922304153442, -0.7399225831031799, 0.3652879595756531, 0.44563841819763184, 0.7746010422706604, 0.8383512496948242, 1.7057490348815918, 0.3114203214645386, 2.433189630508423, 0.08927500247955322, -1.1011415719985962, -0.694034218788147, 1.7775652408599854, -0.6980528235435486, 0.601641833782196, -0.1467127948999405, -1.5181020498275757, -1.345381259918213, -1.0060471296310425, -1.399636149406433, 1.0343397855758667, -1.6138774156570435, 1.0282478332519531, 0.34701311588287354, 1.2619189023971558, 0.3487512171268463, -0.6715561747550964, -0.14877133071422577, 1.3891329765319824, -0.0525742769241333, -0.08712690323591232, 0.691094160079956, 0.23368175327777863, -0.35057947039604187, -0.18448516726493835, -0.550778865814209, 0.7824166417121887, 0.21025456488132477, -0.05543952435255051, -0.10394418984651566, -0.9377331137657166, 0.7869300246238708, 1.2449777126312256, -0.2743140459060669, -1.4853169918060303, -0.7158663868904114, -0.7119858860969543, -0.16463612020015717, 0.7914345860481262, -0.3795924186706543, 0.21487800776958466, -0.44813138246536255, 0.6459055542945862, 1.9129728078842163, 1.0455366373062134, 0.7753812074661255, -1.7790366411209106, 2.0599842071533203, 0.9236314296722412, -0.4134632647037506, 1.4877408742904663, -0.45496752858161926, -0.7024413347244263, -0.39046019315719604, 1.150339126586914, 2.437366485595703, -1.362417221069336, 0.5421556830406189, -0.5581737756729126, -2.6578338146209717, -0.0508490689098835, -1.1859372854232788, -0.10552200675010681, -0.41865038871765137, 0.23995159566402435, -0.23229627311229706, 0.2204907238483429, -1.6476771831512451, 1.0471383333206177, -0.8915256857872009, -0.31492921710014343, 1.257784366607666, 0.2513899505138397, 0.06705103814601898, -0.765555202960968, -1.231956958770752, -0.6516401767730713, 0.7493656277656555, 0.7718386650085449, -0.002633715746924281, 1.4027358293533325, -0.06547550857067108, -1.8070862293243408, 0.9825842380523682, -0.768339216709137, 0.6977944374084473, 1.188372015953064, 0.4037376940250397, -0.8062416911125183, 0.8560969233512878, 0.19639188051223755, -0.06998099386692047, -0.8731562495231628, 0.26524484157562256, -0.9825904965400696, -0.46760714054107666, 0.15417970716953278, -1.3382409811019897, -1.1679035425186157, 1.6249486207962036, -0.8242108225822449, 0.49170827865600586, -1.4817487001419067, 0.12479674071073532, 0.23406726121902466, 1.4413636922836304, 1.775612235069275, 0.2239883542060852, -0.46042823791503906, 0.19766215980052948, -0.42973726987838745, -0.11914880573749542, -2.1392717361450195, 0.900446355342865, -0.10922939330339432, 1.384178876876831, -0.570111870765686, -1.791795015335083, -1.2001008987426758, 0.46684762835502625, -0.5246897339820862, 1.656943678855896, 0.7425655126571655, 0.6389701962471008, 0.5735785365104675, 0.44006049633026123, 0.5109997987747192, 0.5068652033805847, 1.3099942207336426, -0.3337048888206482, -1.3967492580413818, -1.1330766677856445, -0.20471981167793274, -0.16178561747074127, 2.0065958499908447, 0.719085693359375, 0.9831880331039429, -1.1720192432403564, 0.36874920129776, -1.702414870262146, -1.0529083013534546, 0.4475136697292328, 0.053589828312397, 0.023118700832128525, 0.6766555905342102, -0.662801206111908, -1.409165382385254, -1.7812952995300293, -0.712932825088501, 1.0856125354766846, -0.013415304012596607, 0.42345932126045227, 2.4075729846954346, 2.544081687927246, 1.5898985862731934, -0.3846546709537506, 1.3003953695297241, 0.0950385183095932, -0.43381011486053467, 0.0363762341439724, 0.8469683527946472, 1.7204935550689697, 0.548032283782959, -0.11808423697948456, 2.1827430725097656, -0.40467652678489685, 0.09698506444692612, -0.10220599919557571, -0.5792552828788757, 0.009386101737618446, 1.9124115705490112, 0.11491890996694565, 1.0171245336532593, 0.18711444735527039, 1.618626594543457, -1.0281168222427368, 0.4714311361312866, -0.0424470528960228, 0.8949010968208313, -0.14334993064403534, 2.6441755294799805, -1.524911880493164, 0.6657902002334595, -1.6565784215927124, 1.5085334777832031, 0.41242387890815735, -0.1571720689535141, 1.0176279544830322, -0.5826418399810791, 1.218144178390503, -0.08314955979585648, -1.5420801639556885, -0.25267279148101807, -1.1396119594573975, -1.1492923498153687, 0.48363593220710754, -0.3257325291633606, 0.05843158811330795, 0.009446973912417889, 0.18405893445014954, -1.0352598428726196, -0.4646834433078766, -0.48353418707847595, 0.4730735719203949, 1.5814303159713745, -0.6747056841850281, -0.23468847572803497, -1.9626681804656982, 0.4867198169231415, -0.7291206121444702, 1.7181119918823242, -0.4478052854537964, -0.9921804666519165, 1.7865065336227417, 0.7515212297439575, -1.321782112121582, 0.19150836765766144, 0.0025714251678436995, -1.214091420173645, -1.2917273044586182, -0.25432226061820984, -0.535958468914032, -0.18179695308208466, -0.7295461297035217, -1.362411618232727, 1.7474539279937744, -0.3157045543193817, 0.2279158979654312, 0.144158273935318, 0.49634820222854614, -0.23821014165878296, 1.6320610046386719, -0.1447397619485855, 0.5233808755874634, 0.31972986459732056, 0.08301525563001633, 0.03931931406259537, 0.4237395226955414, 0.8918070197105408, -0.38401904702186584, -1.6216500997543335, -1.7542767524719238, 1.0504549741744995, 0.313060462474823, 0.08097214251756668, 0.31763529777526855, -0.9679116010665894, 0.681230902671814, -0.1371062844991684, -0.4285328984260559, 0.4623495042324066, -0.45939475297927856, 0.510227382183075, 0.4461589753627777, -0.6425485014915466, 0.01014406606554985, -1.199233889579773, 0.2336469143629074, -1.5714982748031616, 1.6886430978775024, 0.1813235729932785, 1.053290843963623, 1.8725073337554932, 2.3256518840789795, -0.48666638135910034, 0.21088865399360657, -2.1208667755126953, -0.5067981481552124, 0.2162575125694275, 0.2435622662305832, -1.4754040241241455, 0.8398671746253967, 0.32619690895080566, -0.5167102217674255, 0.9581115245819092, -0.48252397775650024, -0.09170455485582352, -0.016776874661445618, -0.24365496635437012, 0.051566485315561295, 1.3113690614700317, 0.24866338074207306, -1.0965328216552734, 0.4822123944759369, -0.10079791396856308, -0.08076929301023483, 0.8679132461547852, -0.47236719727516174, 2.1804394721984863, 1.1918755769729614, 0.09617909044027328, 0.12743456661701202, -1.564040184020996, -0.8633658289909363, -0.19996760785579681, -1.5068751573562622, -1.1441454887390137, -0.31810319423675537, -2.0002009868621826, -1.3415848016738892, 0.14025598764419556, -0.8124621510505676, -0.9762388467788696, -0.31565654277801514, 0.9310809969902039, -0.4136134684085846, 0.525922417640686, -2.409125328063965, 0.2138095498085022, -0.1888347566127777, -0.8858897686004639, -0.5745644569396973, -0.13271504640579224, -0.973387598991394, 0.7985025644302368, -0.05286231264472008, 1.131287932395935, 0.4505670666694641, 0.46417340636253357, 0.3593289852142334, 0.549228847026825, -1.956108570098877, -0.9190849661827087, -0.07142967730760574, 0.6430938243865967, -0.35273751616477966, -0.47554972767829895, 1.6861499547958374, -1.0504354238510132, -1.0461031198501587, 0.5741181969642639, -0.5908496975898743, -0.7303813695907593, 0.9197618365287781, 0.4900728762149811, -0.17844529449939728, -1.1696661710739136, -0.7115582823753357, 0.13982073962688446, -1.0889337062835693, 0.031320709735155106, -0.825526773929596, -2.440880060195923, -1.6319175958633423, 0.014416570775210857, 0.08473075181245804, -0.00795457512140274, 0.6562404036521912, 0.06737182289361954, 0.08642847090959549, -0.027375968173146248, 1.5192474126815796, 0.5738776326179504, -0.5227206945419312, -0.24737747013568878, -0.36529064178466797, -0.18121083080768585, -0.6873564124107361, 1.2431137561798096, 0.9196334481239319, 1.1916109323501587, -0.9101452231407166, -0.488391637802124, 0.5409734845161438, -0.9487344622612, -1.7512109279632568, -2.425119161605835, 1.2128918170928955, 0.33398890495300293, -1.8484735488891602, -0.7571163773536682, 0.20824015140533447, 0.13836821913719177, -2.3268587589263916, -2.199043035507202, -1.4797136783599854, -1.0853760242462158, 0.22794277966022491, -0.15527191758155823, -0.7313402891159058, 0.48879140615463257, 0.4477638900279999, 1.2675180435180664, 0.14387504756450653, 1.496963620185852, -1.7141368389129639, 0.4731859564781189, -0.20205426216125488, -0.19283722341060638, -0.5870069861412048, 1.1310311555862427, -0.37303227186203003, 0.6613194942474365, -0.18808726966381073, 0.03681635111570358, -1.1284416913986206, 0.06118008866906166, -1.3421666622161865, -0.5885607600212097, -0.907063901424408, -0.4684353768825531, 0.6780295968055725, -0.13246110081672668, -0.26163527369499207, 0.7616775631904602, -0.07809354364871979, 0.15543310344219208, -0.6524487733840942, -0.019859256222844124, 0.22694414854049683, 0.6732572913169861, -0.33917999267578125, 0.27704858779907227, -0.06206542253494263, 1.8456330299377441, 0.7539113163948059, -0.5127764344215393, -0.08283399045467377, 0.6996797323226929, -0.28827863931655884, -0.39758411049842834, 1.2277474403381348, 0.30786019563674927, 0.8122658729553223, 0.537596583366394, -0.7469242811203003, 0.6350092887878418, -0.7804033756256104, -0.04868962615728378, 1.2670749425888062, -0.7171504497528076, -0.7141055464744568, 1.492218017578125, 0.2908894121646881, -1.5300962924957275, -0.6152383089065552, -0.16840118169784546, 1.1030691862106323, 0.584749698638916, -0.0442325621843338, -1.016463041305542, 0.2560093402862549, -0.9962703585624695, 0.6139107942581177, 0.18231983482837677, -0.3187291622161865, -1.2388293743133545, 0.23966705799102783, -0.7120662927627563, -0.4749390184879303, -1.1438266038894653, 1.0937343835830688, 0.796933650970459, -0.5794207453727722, 0.7565569877624512, -0.972415030002594, 0.6144478917121887, -0.9527170062065125, -2.4966678619384766, -0.16745176911354065, 2.492859363555908, -1.4657623767852783, 0.5727623105049133, -0.46782296895980835, -2.6743273735046387, 0.27169740200042725, -0.6012819409370422, -0.7859467267990112, 1.4585645198822021, 0.8206865787506104, -1.1126246452331543, -2.748194694519043, 1.4547280073165894, -1.1607705354690552, 0.2533315122127533, -0.4365329146385193, 1.3874033689498901, -0.438665509223938, -0.5293449759483337, -0.5255632400512695, -0.6505443453788757, 1.4538720846176147, -0.25150391459465027, -1.7483386993408203, 2.9073264598846436, 0.10665437579154968, -0.7458280324935913, 0.29776474833488464, -1.0889486074447632, -0.6205044984817505, 2.464165687561035, -0.4248410165309906, 0.058974798768758774, -1.4210255146026611, -0.1096283420920372, 0.5808354616165161, 0.07875114679336548, 0.9373789429664612, -1.171582818031311, 0.045300714671611786, -0.9081304669380188, 0.014801546931266785, -1.639685869216919, 0.22338874638080597, 2.1885669231414795, -0.9615127444267273, -0.0882156565785408, 0.8353174924850464, -1.0752214193344116, 1.1833640336990356, 0.2067442089319229, 1.9999363422393799, -1.0430493354797363, 0.2908236086368561, 0.2807706594467163, 0.5177685022354126, 0.9803754687309265, -0.8336870670318604, 0.2951097786426544, 0.2214578539133072, 0.4837026000022888, -0.02350636199116707, 0.3712294399738312, -0.9869782328605652, -1.2476979494094849, -0.09364669770002365, -1.2686091661453247, 0.5334550738334656, 0.02757778950035572, 0.9119027853012085, 1.0930570363998413, -0.04517104849219322, 0.7188565731048584, 0.5465055108070374, 1.963732361793518, 0.07984957844018936, 1.1475926637649536, 0.15716882050037384, 0.5125007629394531, 0.3576386272907257, 0.6189885139465332, 0.6856964230537415, -0.3366614878177643, 0.24969889223575592, -2.2065987586975098, 1.1480977535247803, 0.06368658691644669, -0.31077706813812256, -0.710048496723175, -0.7509866952896118, -1.090503215789795, 0.9839491248130798, -0.09862769395112991, -0.2804548144340515, 0.7744408845901489, -0.18786461651325226, -0.6420045495033264, -0.3387576937675476, 0.615715503692627, 0.08669228851795197, -0.4291194975376129, -1.9912790060043335, 0.20543991029262543, -0.5660170316696167, 0.7175753712654114, -0.29600322246551514, -0.08797907084226608, -0.13916829228401184, -2.1699674129486084, 0.9412716031074524, 1.5239317417144775, 0.6056875586509705, 0.9238730669021606, -1.0265341997146606, 0.744941234588623, -1.3413400650024414, -0.9656219482421875, 1.0066996812820435, -2.423525094985962, -0.6334220170974731, 1.0972005128860474, 1.0354244709014893, -1.4049484729766846, 1.7843074798583984, -0.0338251069188118, -1.6917814016342163, 0.3364139199256897, 1.949062466621399, 0.4071449041366577, -1.424699306488037, 0.8048247694969177, -0.37259939312934875, 2.3597898483276367, -1.4213746786117554, -0.5995897054672241, -0.786133348941803, 0.11501777172088623, 2.4680135250091553, -1.6133363246917725, -1.0465337038040161, 0.06901713460683823, 0.917600154876709, 0.008053446188569069, 0.8447092771530151, 1.5237351655960083, -0.7368707060813904, -0.7093976736068726, 0.6873316168785095, -0.3275294005870819, -0.5235018730163574, -0.8594006896018982, 0.5869925618171692, 1.825061321258545, 0.1569133698940277, -0.279544860124588, 0.24361146986484528, 0.7635154128074646, 0.1395098716020584, -0.8926423788070679, -0.6049506664276123, -2.5666561126708984, -0.8085012435913086, 1.304246187210083, -0.17815077304840088, -0.6572540402412415, 0.7492507100105286, -0.7499600052833557, 0.6181150674819946, -1.3278590440750122, 0.9137468338012695, 1.7429296970367432, -0.32341521978378296, 1.4608640670776367, 0.14871850609779358, -1.2699933052062988, -0.9689104557037354, 0.20998895168304443, -0.87633216381073, -0.23304486274719238, 1.4028306007385254, -0.5392530560493469, 1.3670629262924194, -0.7182955145835876, -1.4584399461746216, -1.5998982191085815, -0.34164857864379883, 0.6664136052131653, -0.37122228741645813, -0.4194646179676056, -0.3247259855270386, -0.2135113924741745, 1.1549654006958008, -1.0128204822540283, 0.44195082783699036, -0.3403164744377136, 0.8614231944084167, -0.593699038028717, -0.5570189356803894, -0.17329154908657074, 0.4200564920902252, 0.0254502035677433, -0.09622789174318314, -2.3266754150390625, 0.07218130677938461, -0.08015225827693939, -0.7191705703735352, 0.9200379848480225, -1.8737884759902954, -0.4556686580181122, -1.9888713359832764, 0.613798975944519, 0.8208550214767456, -0.8130265474319458, -0.36281919479370117, -2.6221134662628174, -1.960846185684204, -0.3465319573879242, -1.310652256011963, 1.5562312602996826, 0.7993676066398621, -1.482729434967041, -1.4969619512557983, -1.247420310974121, -0.6769253015518188, -0.281734824180603, -0.917079508304596, 0.28492289781570435, 1.1210618019104004, -0.40440285205841064, 0.8076744675636292, 1.304933786392212, 0.6964664459228516, 0.4439423084259033, 0.1642102301120758, 0.4619392454624176, -0.4659118354320526, -0.2793569266796112, -0.03256338834762573, 1.719648003578186, -1.958343267440796, 0.1061893031001091, 0.7030957341194153, -0.2751028537750244, -0.7140710949897766, -1.362337589263916, 1.6049309968948364, -1.4661571979522705, -0.430370569229126, 0.5996258854866028, 1.1891528367996216, 0.2455444037914276, 1.1389189958572388, 1.7585121393203735, 0.48811614513397217, 0.7425596714019775, 0.5689985752105713, -1.868883490562439, 1.0382784605026245, 0.7134681940078735, 0.03384657949209213, 0.7500548362731934, -0.675625741481781, -0.8923426270484924, 0.41334909200668335, 0.7622436285018921, 0.0029348013922572136, 0.7774820923805237, -0.001531598623842001, -0.014763562940061092, 0.08677195757627487, -0.5379765033721924, 0.23968026041984558, 0.9630336761474609, -0.1346307098865509, -0.35724130272865295, -0.41417625546455383, 0.7234784364700317, -1.7103073596954346, -1.104564905166626, 1.4614975452423096, 0.6337489485740662, -1.2641116380691528, -0.8847958445549011, 0.8327606320381165, 0.8779584169387817, 0.29554855823516846, -0.14838816225528717, -1.4664510488510132, -1.9033150672912598, -0.5620403289794922, -2.3439931869506836, 0.1239657774567604, -2.9944727420806885, -0.8739832639694214, 0.20091849565505981, -1.2499749660491943, -0.05901603028178215, -0.6527774333953857, -1.797437310218811, 0.1991778165102005, -0.8002843856811523, 0.3937239348888397, 0.7334922552108765, -0.9600404500961304, -0.1801702380180359, -0.06709101796150208, -0.030758239328861237, 1.7729192972183228, -1.80534827709198, -1.2232275009155273, -0.35414430499076843, 0.940756618976593, 0.1167752742767334, 1.580844759941101, -0.4464794397354126, 2.6805338859558105, 0.6250583529472351, 0.845947265625, 0.2961536645889282, -0.46141621470451355, 0.7344374656677246, -0.9137884378433228, 2.1028544902801514, -1.0295345783233643, 1.0446468591690063, -0.3777461349964142, -0.34565362334251404, -1.4609347581863403, -0.5819214582443237, 0.7397715449333191, -0.142076775431633, -1.2010548114776611, -0.5917904376983643, 0.9263724088668823, -0.36097124218940735, -1.2982628345489502, -0.843744695186615, -1.1115643978118896, -0.5294094681739807, 0.5439339280128479, 1.4757192134857178, -0.8516601920127869, -0.1744024157524109, -0.9706295728683472, 2.1165075302124023, -0.30141231417655945, 0.8392590880393982, 0.791413426399231, -0.8138260245323181, -1.2599914073944092, -0.47157156467437744, -0.5122830271720886, 0.08748453110456467, -0.7069615125656128, -0.8150736689567566, 1.0619127750396729, -0.020380910485982895, -1.5683180093765259, 0.17069102823734283, 1.1040765047073364, 0.1348157674074173, 0.7346704006195068, -0.09655322134494781, 0.3899998664855957, -0.965025782585144, -0.9692181944847107, -0.4946954846382141, -0.48886266350746155, 0.3125612139701843, 0.4054350256919861, -0.38216784596443176, -0.8915554881095886, 0.39072543382644653, -0.8408716320991516, -1.3908635377883911, -0.3074158728122711, 0.5752478241920471, 1.1473017930984497, -1.0189378261566162, -1.493107795715332, 0.16278046369552612, 1.0489403009414673, -0.17609967291355133, 0.5611390471458435, 0.9113673567771912, 1.5004191398620605, 0.0898326113820076, -0.2852829098701477, -0.011219545267522335, 0.34005922079086304, -0.034264981746673584, 0.5009467005729675, -0.6506373286247253, 0.34551748633384705, -0.5801323652267456, 0.21524223685264587, 0.9381205439567566, 0.481595903635025, -0.5781311392784119, 0.23027589917182922, 0.2805667817592621, -0.010165734216570854, 0.025939226150512695, -1.710829257965088, -0.5696302056312561, 2.644740581512451, -0.8605528473854065, -0.7801156044006348, -0.9332476854324341, -0.4614032506942749, -0.6051169037818909, -0.8431288599967957, 0.16608303785324097, 1.017056941986084, -0.5451931953430176, -1.7193005084991455, 0.4434066116809845, 0.7014788389205933, -1.188008189201355, 1.443751573562622, -0.07472412288188934, 0.8821538686752319, 1.7581231594085693, -1.227388858795166, -1.094307780265808, 1.9482156038284302, 0.6045277118682861, 0.5307283401489258, 1.2247401475906372, -1.2003384828567505, 0.5664626359939575, 0.5277214050292969, 0.3268754780292511, 0.3916572630405426, 1.5677268505096436, -0.8759137392044067, 0.745265781879425, -0.09284639358520508, 0.0817091315984726, 0.19280162453651428, -0.70302414894104, 1.3735830783843994, 0.7108460068702698, 0.6218093633651733, 1.6166757345199585, 1.3152631521224976, -1.659878134727478, 0.1375252604484558, 1.0942957401275635, 1.3073490858078003, -1.018631935119629, -1.4180846214294434, -1.0026366710662842, 1.401068925857544, -0.1887342780828476, -0.14780612289905548, -0.4712778627872467, 0.7888428568840027, -0.44928550720214844, 0.4036098122596741, -0.24860265851020813, 0.5411360859870911, 0.7386093139648438, -1.0242871046066284, 0.029336094856262207, 0.8688750863075256, 0.39885810017585754, 1.5347228050231934, -0.08023674786090851, 0.42151275277137756, 0.4185146689414978, 0.1107230931520462, 0.19228850305080414, 1.267408847808838, -0.020730214193463326, -2.1142940521240234, -0.6819567680358887, 0.13696859776973724, -0.055485423654317856, -0.9343343377113342, -1.1484485864639282, -0.4593878984451294, 1.6923075914382935, 1.3235902786254883, 1.533492922782898, -0.51969975233078, -0.15758270025253296, -1.7698235511779785, -0.20684576034545898, 0.7269222736358643, -0.3691951334476471, -0.7926304936408997, -0.640339732170105, 0.49220991134643555, -1.0072505474090576, -0.2411353439092636, 0.7246414422988892, -2.370772123336792, -0.3804512023925781, -0.6002875566482544, -1.0770145654678345, -0.8817876577377319, 0.2186259776353836, 1.128156065940857, 1.3354260921478271, -0.019405707716941833, 0.2863624691963196, 0.3739657402038574, 0.3984288275241852, -1.12061607837677, -0.44111618399620056, -0.8664275407791138, -0.8658305406570435, -0.18964597582817078, -0.5322882533073425, -0.9518770575523376, -1.3180516958236694, 1.2314085960388184, 0.34480059146881104, 0.5079017281532288, 0.921408474445343, 1.307631015777588, -0.41213124990463257, 0.6971607804298401, 0.19709837436676025, 0.9267571568489075, 0.5433476567268372, 0.6951572299003601, 0.9500935077667236, 0.9987280964851379, 1.8117810487747192, 1.104810357093811, -1.849183440208435, -0.5686600208282471, 0.6068320870399475, 1.097970724105835, -0.7937785387039185, 0.6141589283943176, -2.3934552669525146, 0.735507607460022, 0.2978156507015228, -0.40469595789909363, -0.25026339292526245, 0.4814116656780243, 0.6796557307243347, -0.13258527219295502, 0.31330129504203796, -0.08921831846237183, 0.759743332862854, -0.9329321980476379, -0.6284998059272766, -1.7328470945358276, -2.075735330581665, 0.21337316930294037, -1.8097118139266968, 0.04028059542179108, 0.41435229778289795, 0.2353862226009369, 1.2522741556167603, -1.9036332368850708, -0.9269984364509583, -1.536293387413025, -0.39532023668289185, 0.8919316530227661, 1.696359634399414, -0.7103430032730103, 0.24938002228736877, 0.2308829426765442, 0.17759522795677185, -0.269246369600296, -0.051622092723846436, 0.5088396668434143, 0.07824315130710602, -1.244836688041687, 1.9871327877044678, -0.541171133518219, -0.04904654994606972, -0.2762618660926819, 0.6320489048957825, -0.9133285284042358, -0.030045144259929657, 0.3870067596435547, -0.07571938633918762, -0.060570672154426575, 0.5942161083221436, 0.7724485397338867, 0.2810300290584564, -0.4285883903503418, 0.9818629622459412, -1.3687607049942017, 1.7122896909713745, -0.0029384924564510584, 2.8881423473358154, -0.19275206327438354, 1.626656174659729, -0.7737019658088684, -0.6748464107513428, 1.1501820087432861, -0.7742026448249817, -0.048105642199516296, -0.10395798087120056, 0.26128819584846497, 0.09375660866498947, -0.3170553147792816, 2.0194907188415527, -1.3141738176345825, 1.0530866384506226, 1.4123541116714478, -0.037052471190690994, 0.3469170331954956, 0.7079806923866272, -1.2676234245300293, 1.9983112812042236, 1.2790805101394653, 0.1370587795972824, 0.8537300825119019, 1.7030279636383057, -2.0739939212799072, -0.5298082828521729, -0.013874969445168972, -0.5500586032867432, 1.2424800395965576, 0.5415905714035034, 0.4637843072414398, -0.34076249599456787, -1.0645136833190918, 0.06729702651500702, -2.6783998012542725, -1.0262664556503296, 2.435112714767456, -0.03069448657333851, -0.06382452696561813, -0.13109813630580902, -0.7000311017036438, -0.25556087493896484, -1.0149372816085815, 1.850310206413269, -2.1343486309051514, 0.5211694836616516, -0.3097142279148102, 0.13858769834041595, -0.19959308207035065, 0.06206797808408737, 1.494206190109253, 1.0875149965286255, 1.532297134399414, -0.7679157853126526, 2.229888677597046, -0.9311254620552063, 1.113871455192566, 0.616530179977417, -0.004427290055900812, 0.5627582669258118, -1.1630605459213257, -0.9685155749320984, -0.8805447816848755, -0.0449976921081543, -1.3106781244277954, -0.49977612495422363, -0.5849545001983643, 0.4833105802536011, -1.109262466430664, -0.5759788155555725, 0.053768593817949295, 1.6984249353408813, 1.624640941619873, 1.6904208660125732, 1.5959957838058472, -1.5269850492477417, 0.9725285768508911, -2.269874334335327, 0.36937612295150757, -0.7075634598731995, -0.45702940225601196, 0.5117883086204529, -2.440711259841919, 0.7360222339630127, -0.5675471425056458, 0.8563582897186279, -0.9369951486587524, 1.4316965341567993, -0.8372393846511841, 0.7513319253921509, 0.009051729924976826, -0.12600615620613098, 0.6833391785621643, -0.6822142004966736, 0.07748040556907654, -0.7748541831970215, -0.190144881606102, 0.9359495043754578, -0.45517224073410034, 1.295717477798462, 1.2998186349868774, -1.5677173137664795, 1.215709924697876, -1.9072816371917725, 0.4095321297645569, -0.7346646785736084, -0.30027732253074646, 0.46639108657836914, 1.210343360900879, 1.1535993814468384, 0.14658759534358978, 0.3829960823059082, -0.6063032150268555, -1.3400218486785889, -0.10665059089660645, -0.5225383639335632, -1.0200260877609253, 0.9683477282524109, -0.47339990735054016, 0.1929025650024414, 1.1449817419052124, 0.44191160798072815, -0.25380730628967285, 0.9298605918884277, -1.1578518152236938, -2.8360283374786377, -0.6912917494773865, -0.802643895149231, -0.3359838128089905, 1.7932403087615967, -1.0314842462539673, 0.5015031695365906, 0.7032886147499084, 0.82866370677948, 0.13169044256210327, -0.9494364261627197, 0.951374351978302, 0.5937926173210144, -0.37591642141342163, 2.2905824184417725, -0.9165979027748108, 0.417020708322525, 0.5419034361839294, -0.5000507235527039, 0.4425969123840332, -0.6282017230987549, 0.9130924940109253, 0.0003598319599404931, -0.6541798710823059, 0.8539979457855225, 1.6071507930755615, 0.03478217497467995, -0.265195369720459, -0.6587516665458679, -0.3901340663433075, -0.33821046352386475, 0.005037774331867695, -0.35380685329437256, -1.082162618637085, 0.14277298748493195, 1.2085808515548706, 0.9678901433944702, -0.8024027347564697, 2.0780069828033447, -0.43493765592575073, -1.0917363166809082, 0.5038598775863647, -0.4361472725868225, -0.4766608476638794, 0.8173820972442627, -0.11248476058244705, -0.8284227252006531, -0.531776487827301, 0.5559398531913757, 2.6162233352661133, -0.04604983702301979, 0.36165735125541687, -0.8028031587600708, 1.1973334550857544, -0.699517011642456, -0.6807140707969666, -1.1444766521453857, -0.8099958300590515, 1.043965220451355, 0.5285124778747559, 0.3930993974208832, 0.4915434420108795, -1.8961588144302368, -0.5721983909606934, 0.3594939112663269, -1.8510762453079224, -1.61692214012146, -0.003270852379500866, 1.522709608078003, -0.0420360192656517, -0.9223323464393616, -0.8289685249328613, 0.40491458773612976, -0.24482332170009613, 0.43270838260650635, 1.1289483308792114, -2.3730061054229736, 0.24281813204288483, -0.6425946950912476, -0.6122610569000244, 0.9771450161933899, -0.6842504143714905, 0.17346838116645813, 1.1884469985961914, 0.1593376249074936, 0.06113586574792862, 0.39505642652511597, -0.3180231750011444, 0.21198293566703796, -0.0008494300418533385, 1.7869117259979248, 0.02138010412454605, 0.06822420656681061, 0.5800442099571228, 1.0184706449508667, 0.19647204875946045, 0.8694356083869934, 0.5368568301200867, 0.1418185532093048, 0.030470600351691246, 0.5387231707572937, -0.4616173207759857, 0.042190056294202805, 0.6461716294288635, -1.3301175832748413, -0.6054136157035828, -0.17064589262008667, -2.0661368370056152, -0.1345096379518509, -0.42519503831863403, 0.6669784188270569, 2.9197030067443848, -1.1864745616912842, -0.6327341198921204, -0.5928439497947693, -0.4454183280467987, -0.4377542734146118, 0.3442081809043884, -0.5061838626861572, -1.791582703590393, 1.66067636013031, 0.025319654494524002, 0.18156610429286957, -0.6243569254875183, -0.5950295925140381, 0.03250214084982872, 0.3205893337726593, 0.27967050671577454, -1.2176798582077026, 2.47556471824646, -0.6211009621620178, 2.0550875663757324, -1.1361130475997925, -1.396084189414978, 0.6010512113571167, 0.6386415362358093, -0.960332989692688, -2.7847776412963867, -0.492343932390213, 1.4367012977600098, -0.3008633852005005, 0.8803421854972839, 0.874817967414856, 1.7525030374526978, 0.7698215246200562, -1.1317282915115356, -0.5511919856071472, 0.08146088570356369, 1.0857685804367065, 2.116708755493164, 1.877762794494629, 1.0015381574630737, 0.8580464720726013, -1.3406473398208618, 2.5499911308288574, -0.9538933634757996, -0.893841028213501, 1.859535813331604, -0.9489690065383911, 0.7102315425872803, -1.0010353326797485, 0.8959792852401733, -0.69132000207901, -0.308238685131073, 0.3047942519187927, 1.280874490737915, 0.5379163026809692, 0.0924585834145546, 0.8093422651290894, 0.3713393211364746, -0.31130534410476685, 1.5774924755096436, -1.6584454774856567, 0.350528746843338, 0.8822699189186096, 0.3349898159503937, 1.3324863910675049, 1.0862298011779785, 0.07246947288513184, -0.7382684946060181, 0.1486598253250122, -1.2360687255859375, 0.5905455350875854, 2.791386127471924, -0.41681697964668274, 1.8743857145309448, 1.4920557737350464, -0.47819286584854126, -1.5749741792678833, -1.1057101488113403, 0.38798990845680237, -0.7514136433601379, -0.03920098766684532, -1.1217166185379028, 1.1121610403060913, 0.3028416335582733, 0.5216057896614075, -1.4775856733322144, 0.9416400194168091, 0.1547509729862213, -0.6156625747680664, -0.08726713806390762, -0.07095950096845627, 0.38725537061691284, 1.7136532068252563, 0.7765929102897644, -0.5541298985481262, 0.44948679208755493, 1.33174729347229, 0.055280499160289764, -0.3838019073009491, 0.26947903633117676, -0.6930265426635742, -1.3140439987182617, -0.37733033299446106, -0.6054858565330505, -0.8594849109649658, -0.6041701436042786, 1.9163708686828613, 0.19622327387332916, 0.7672082185745239, -1.5417226552963257, -1.145700454711914, -0.16514821350574493, -0.15657521784305573, -0.04684621840715408, -0.5904092192649841, 1.026774525642395, -0.6531065702438354, -0.1835688054561615, 2.055604934692383, 1.1165540218353271, 1.039210319519043, 1.8257118463516235, -1.1607170104980469, 0.4179880917072296, -0.5416433811187744, -0.5957909226417542, 0.06326822191476822, 0.5079178810119629, 1.9280579090118408, -1.8034253120422363, 0.877754271030426, -0.8497403860092163, -0.22001279890537262, -0.4171844720840454, 0.836024820804596, 1.9052056074142456, -0.39141571521759033, 0.19698211550712585, -0.22061648964881897, 0.5114534497261047, -0.5083719491958618, 0.05554085597395897, 1.0818321704864502, 1.4076608419418335, 0.7169991731643677, 0.7208683490753174, -0.7105146646499634, 0.6992643475532532, 0.3100121021270752, 2.209683656692505, 0.6820157766342163, 0.39851298928260803, -0.794298529624939, -0.5561339259147644, -0.008558303117752075, -0.48578834533691406, 0.12857849895954132, 0.5622798204421997, 1.7802801132202148, -2.210310935974121, 0.9641214609146118, 1.7765076160430908, 1.4621318578720093, 0.6422252655029297, 0.2379940152168274, -0.009486522525548935, 0.9331762790679932, -1.669228196144104, -0.173758402466774, -0.15824705362319946, 0.04773041605949402, 0.8638896346092224, -1.3908153772354126, -1.6770893335342407, 0.43909189105033875, 0.380874365568161, 0.7531099319458008, -0.7958866357803345, -0.24516049027442932, 0.27277567982673645, -0.2955380082130432, 1.8817843198776245, 1.3972538709640503, 1.4156222343444824, 1.335628867149353, 0.8448820114135742, -0.12770971655845642, 0.11078080534934998, 0.18308588862419128, 1.4668172597885132, 1.0357493162155151, 0.6106020212173462, -0.14320597052574158, 1.8545705080032349, -1.4277465343475342, 2.025186777114868, -0.697227954864502, -0.4588959217071533, -0.656003475189209, -0.4277177155017853, 0.09685930609703064, -0.20986208319664001, -0.5435588359832764, 1.5346789360046387, -1.4217418432235718, -1.1558810472488403, 0.23547570407390594, 0.2804095447063446, -0.573836624622345, 0.11653858423233032, -0.00865850318223238, -0.3299387991428375, -0.788645327091217, 1.6897433996200562, 0.9780598282814026, 0.19982632994651794, 0.8428426384925842, 0.1783301681280136, -0.5444340705871582, 0.5196818113327026, 0.8398171663284302, -1.5357139110565186, -0.17247258126735687, -0.7649048566818237, -1.4018783569335938, -0.7601856589317322, 0.41797757148742676, 1.1534231901168823, 1.9509620666503906, 0.7487524151802063, 0.474588006734848, -1.3616137504577637, -0.5300245881080627, -0.5425361394882202, 0.7493546009063721, -0.15500439703464508, -1.1493051052093506, -0.016597451642155647, 0.2277286946773529, -0.15438689291477203, 1.1065449714660645, 2.6428399085998535, 1.2410448789596558, -1.6809865236282349, 0.41860851645469666, -0.7642599940299988, -0.5624911785125732, 0.9167126417160034, 0.6524022221565247, -2.2647368907928467, 0.0793636292219162, 0.13304950296878815, 1.8970515727996826, 0.45404869318008423, -0.24205654859542847, -0.28485575318336487, 0.24870088696479797, 0.5908371806144714, -0.3681958317756653, -0.28308171033859253, -0.5937649011611938, -0.8031137585639954, -1.0430185794830322, -1.0152289867401123, 1.34623122215271, -0.6164308190345764, 0.4138604700565338, -0.26731908321380615, 1.129815936088562, -0.8266799449920654, -0.2532501816749573, 0.8133373260498047, -0.18181636929512024, -1.005252718925476, -0.5705198049545288, -1.0720219612121582, 0.22710593044757843, -0.7181695103645325, -1.568183183670044, -1.541683554649353, -1.90684175491333, 0.5328527092933655, 0.72254878282547, 1.778555989265442, -0.03203657269477844, 0.025636568665504456, 1.3338963985443115, -0.38879236578941345, -0.9967032670974731, 1.0041519403457642, -1.5727508068084717, -1.3748135566711426, -1.810145616531372, 0.37526780366897583, -0.012052896432578564, -1.2299472093582153, -0.9856221675872803, -1.2173839807510376, 0.9916951060295105, -0.03487100824713707, -0.7371827960014343, 0.5113565325737, -1.5307412147521973, 0.3622722625732422, -0.16831566393375397, 0.6937896609306335, 1.6578716039657593, -0.02840660884976387, -0.013598307967185974, 0.5103966593742371, 0.2128283977508545, -0.9539494514465332, 1.7800167798995972, -0.2692827880382538, 1.15144944190979, 1.5465604066848755, -0.5923826694488525, -0.3148058354854584, 0.1521875560283661, -0.8001481890678406, 0.7057386636734009, -1.6191620826721191, -0.7642089128494263, -2.7103192806243896, -0.8047756552696228, 0.6523630619049072, -0.33851200342178345, 2.9599359035491943, 1.1334099769592285, 0.8377522230148315, -1.1911698579788208, 0.06702783703804016, -0.1146506518125534, 1.661630630493164, -0.15484745800495148, 1.1393256187438965, 0.847667396068573, 0.5254283547401428, 0.4818449020385742, -0.4538830518722534, 0.17130424082279205, 1.1173344850540161, 0.4920264780521393, -1.086026668548584, -0.6095885038375854, -1.6152437925338745, 0.1930703967809677, -0.3469596207141876, -0.5319743752479553, 0.9424425363540649, 0.33315208554267883, -0.9217430949211121, 0.5928747653961182, 1.2704445123672485, -0.9500337839126587, -1.5216846466064453, 1.0561718940734863, 0.817487895488739, -0.615972638130188, 0.21129882335662842, -1.2258936166763306, 0.9654237031936646, 0.26334473490715027, -1.5662131309509277, -0.12171842902898788, -0.1438494622707367, -0.30561336874961853, -1.7086660861968994, -0.06574781984090805, -0.6231687664985657, -1.4444003105163574, -0.3670642375946045, -0.02626878395676613, -1.2829326391220093, -0.33268100023269653, 0.26772481203079224, 0.22296838462352753, 2.483109712600708, -0.22613893449306488, -0.7329057455062866, 0.07568417489528656, -0.857232391834259, 0.21577614545822144, -0.890449047088623, 1.152612328529358, -1.5335983037948608, -0.19490371644496918, -1.6771446466445923, 1.1352791786193848, -0.1746513694524765, -0.2357439398765564, -0.1007608100771904, 0.2184169441461563, 0.9536301493644714, 1.0773721933364868, -0.88108229637146, -0.7100957036018372, -0.4729587137699127, -0.3465745747089386, -0.9996259212493896, 1.8423056602478027, -0.10843899846076965, 1.5072754621505737, -0.7455973029136658, -2.165635347366333, -1.7535520792007446, -0.8903127908706665, -1.3821927309036255, 0.2678791284561157, 1.478535771369934, 0.1558387577533722, 0.5951386094093323, -0.20450788736343384, 0.34120601415634155, 0.1344103366136551, -1.905199646949768, -0.8515423536300659, 0.051266372203826904, -0.3182719945907593, -0.4536985754966736, 1.500091314315796, 1.4944158792495728, -0.21449366211891174, 0.03456513211131096, 1.7115461826324463, 0.0866965800523758, -0.12913301587104797, 1.7730869054794312, 1.337806224822998, 0.423164039850235, -1.1657822132110596, -0.6535623669624329, 0.7133485674858093, -0.8245710730552673, 0.5314528346061707, 0.7879390716552734, 2.6408398151397705, -1.380623459815979, 0.8820834755897522, -0.4594215452671051, 0.2907150983810425, 2.588907241821289, -1.5701406002044678, -0.4000626504421234, 1.1018704175949097, -0.3900519013404846, -0.6241305470466614, 0.6683303713798523, -1.523785948753357, 0.6252982020378113, 0.15840314328670502, 1.3973218202590942, 0.9853507280349731, 2.244758129119873, -0.7846077084541321, 1.0687419176101685, -0.9995378255844116, -1.8764134645462036, 0.275409996509552, 1.0840399265289307, -0.9662953019142151, -0.25917840003967285, -0.18121954798698425, -0.3831436336040497, -1.174493432044983, -0.7009239792823792, 2.0627126693725586, 0.22294795513153076, 1.1638745069503784, -0.3844005763530731, -0.896876871585846, -2.813425064086914, -1.0903006792068481, 1.6316581964492798, -0.775668203830719, -1.7632008790969849, 1.243396520614624, 1.597050428390503, -1.0538018941879272, -1.252277135848999, 1.4528640508651733, -0.7657304406166077, -0.4565710425376892, 0.8373238444328308, 0.8080039024353027, 3.0525240898132324, 0.34059634804725647, 0.8068632483482361, 1.333404541015625, -0.2865820527076721, -0.13661912083625793, 0.570401132106781, 0.2847422659397125, -0.5121538043022156, 0.31467241048812866, 0.036925457417964935, 0.29170891642570496, -0.4041300117969513, 1.0717524290084839, -0.006459979340434074, -1.0157052278518677, 0.2528146505355835, 1.0407567024230957, -0.022301381453871727, 0.32378166913986206, -0.6478648781776428, 2.430386781692505, -0.2999384105205536, 0.001730015967041254, -1.5629242658615112, -1.2225370407104492, -1.0204957723617554, -0.10172539204359055, 0.5341545343399048, -0.8843221664428711, -0.8427640795707703, 0.22298161685466766, 0.23310473561286926, -0.3902413845062256, 0.6129907965660095, -0.0317288301885128, -0.1405213624238968, 1.0085432529449463, 1.6393083333969116, -0.2182379513978958, -1.316763162612915, -0.7240327596664429, 0.6178697943687439, -1.7207907438278198, -0.8335520029067993, 2.1533584594726562, 0.5830882787704468, 1.6570875644683838, 0.5163024663925171, 1.1933523416519165, -0.27113425731658936, 0.7263817191123962, -1.541520357131958, -0.1948700100183487, -0.1364799290895462, 1.3562376499176025, 0.49677184224128723, 1.1079031229019165, -1.195902943611145, 0.026991361752152443, -0.8524424433708191, 2.3565709590911865, -0.09544163942337036, -0.9293582439422607, -1.7786431312561035, -1.4906262159347534, -2.250396728515625, -3.187225341796875, -0.16804514825344086, -1.048108696937561, 1.0576848983764648, -1.1288526058197021, -0.38987302780151367, -0.4570312201976776, -1.9980047941207886, -0.9131236672401428, -0.6451573371887207, -2.765306234359741, 0.2915134131908417, 0.43888673186302185, 0.08634362369775772, 1.7346069812774658, 0.1874939501285553, 0.6676462292671204, 0.8268274068832397, -1.0125155448913574, -0.5556381940841675, 0.060807038098573685, 1.4901347160339355, 0.8774152398109436, 0.4944227635860443, 0.13127253949642181, 1.4018532037734985, -1.4597952365875244, -1.23068368434906, -2.3334407806396484, 0.6968809962272644, -0.22168852388858795, 0.1422693133354187, -1.2231829166412354, 0.48285409808158875, -1.7126438617706299, -1.5050691366195679, 1.6553573608398438, 2.314826726913452, 0.9266237616539001, -0.7097041010856628, -0.5606879591941833, -0.7465355396270752, 0.7653451561927795, -1.0800853967666626, -0.20601961016654968, -0.2723110318183899, -0.6092667579650879, 0.9259184002876282, -0.9549956917762756, 1.6088738441467285, 1.2778282165527344, -0.24641655385494232, 1.1040273904800415, -0.7875440716743469, 1.1270413398742676, -0.37623417377471924, -0.018001509830355644, 0.031305938959121704, -0.22282017767429352, -0.043972618877887726, 1.686851978302002, -0.8591638207435608, 0.41232845187187195, 0.8857847452163696, 0.9811956882476807, 0.42240920662879944, 0.08381905406713486, 0.26008686423301697, -1.4567632675170898, 2.4832911491394043, -0.581155002117157, -1.1307355165481567, -0.8290140628814697, -0.020586583763360977, 1.035345435142517, 0.31223201751708984, 0.33950045704841614, 0.10617814213037491, -0.9296039342880249, -1.6444429159164429, 0.04094929248094559, 0.09683013707399368, 0.9565720558166504, -0.524608314037323, -0.4095931649208069, 0.5084158778190613, 0.10727176070213318, -0.040242914110422134, -0.9278952479362488, -0.2241731435060501, -0.6326072216033936, 0.28270038962364197, 0.8499491810798645, 0.6198142766952515, -0.8087584376335144, 0.9832215905189514, -1.6271474361419678, -0.28562045097351074, -0.6084781289100647, 0.2391631156206131, 0.6439118385314941, -0.22661830484867096, -0.6815073490142822, -1.275829553604126, -0.691287636756897, 1.1122792959213257, 1.239983081817627, 0.16432419419288635, 0.1354517936706543, 0.02011486329138279, -1.0011054277420044, -1.0154225826263428, -0.5989964604377747, 1.0707167387008667, -0.8636423945426941, 0.10222195833921432, 0.5993233919143677, 2.5084404945373535, 0.2482202649116516, -2.0604159832000732, 1.09821617603302, -0.10040413588285446, 0.8364492058753967, 0.023545075207948685, 0.3447624146938324, -1.0188908576965332, 0.6487506628036499, -2.0531766414642334, -0.80213463306427, 0.5786415338516235, 0.04584820568561554, 1.4546384811401367, -0.08867727965116501, 0.5830293893814087, 0.720064640045166, 1.0326180458068848, -0.07209160923957825, 0.039540354162454605, 0.20496582984924316, 1.0471915006637573, -1.585214376449585, -0.7486633062362671, -2.1509687900543213, -2.4805712699890137, -0.5454192757606506, 0.7395628690719604, 0.7960371971130371, 0.694843053817749, -0.4322781264781952, 0.540296196937561, -0.5018757581710815, -0.20959939062595367, -0.9777867794036865, 0.08832339942455292, 0.9007872343063354, 0.9906582832336426, 1.3344348669052124, -0.02465011551976204, -0.3276539146900177, 0.7595558762550354, 0.5768884420394897, 1.0290018320083618, -0.06781674176454544, -0.8971107602119446, 0.47530558705329895, 0.4322834014892578, -0.8487904071807861, -1.1779793500900269, -1.7983267307281494, 0.1025126576423645, 0.5620782971382141, -0.26883554458618164, -1.1877354383468628, 0.816676139831543, -1.1259065866470337, -0.9094559550285339, -0.2364474982023239, 0.4958901107311249, -0.1337125152349472, -0.9426305294036865, 0.9456433057785034, 0.23416119813919067, -0.813105583190918, 0.6110126376152039, 0.3081616163253784, -0.23587313294410706, 0.12869392335414886, 1.0273258686065674, -0.4429261088371277, -0.6409032344818115, 0.23936310410499573, 0.31431981921195984, 0.36287179589271545, 0.30667179822921753, -0.10509143024682999, -1.1574805974960327, 1.1867995262145996, -0.7382765412330627, -1.5108346939086914, 1.556515097618103, 0.26731935143470764, 0.8371903300285339, -0.774964451789856, 0.16448712348937988, 0.33678972721099854, -0.372661828994751, -0.19738024473190308, -2.475536823272705, -0.41253790259361267, -1.158400297164917, -0.4298476278781891, -0.9844710230827332, -0.18766579031944275, 0.454280287027359, -1.2766631841659546, 0.8680893778800964, 1.7741068601608276, 0.0939808040857315, 0.5854058861732483, 1.118534803390503, -0.15874917805194855, 0.7686217427253723, 0.6213660836219788, 0.8762837052345276, 0.1923111379146576, -0.26026418805122375, -2.4967217445373535, -0.9147235751152039, 2.2594001293182373, -0.3345057964324951, -0.5358847379684448, -0.6607592105865479, 0.08374164998531342, 0.49937379360198975, -0.028472963720560074, 1.1025843620300293, 0.8514940142631531, 0.6741422414779663, -0.3918338119983673, 0.10606468468904495, -1.2471559047698975, -0.5273138880729675, -0.6907951235771179, -0.1220814436674118, -1.098089337348938, -0.4550679624080658, -1.2599884271621704, 2.3294529914855957, -0.8030516505241394, -0.8610643744468689, 0.7446749210357666, -0.3251897394657135, 0.1303374171257019, 0.969230592250824, -0.9513905644416809, 0.959264874458313, 0.6663058996200562, -0.9563153386116028, -1.0420702695846558, -0.9043719172477722, -1.5571355819702148, 0.36940380930900574, 1.4834266901016235, 1.248660683631897, 0.553896963596344, 2.4991326332092285, -1.5281902551651, -1.1670209169387817, -0.19878217577934265, 1.9003186225891113, 0.08701247721910477, 0.9208624958992004, -0.014077618718147278, -0.8680838346481323, 0.1492946445941925, -1.5692270994186401, -0.6528509855270386, -2.1220831871032715, -1.0984441041946411, -1.760717511177063, -1.4690557718276978, 1.303012490272522, 0.05399223044514656, -0.002125352155417204, -1.9388841390609741, 1.3092594146728516, 0.14669741690158844, -1.231791377067566, -1.1664360761642456, 0.773149311542511, 0.04107893630862236, 0.23996269702911377, 0.2691577672958374, 0.7729090452194214, 0.48296254873275757, 0.3973701298236847, -0.27818045020103455, -0.4409400522708893, 1.1129738092422485, -0.5928699374198914, -0.13960742950439453, -1.267344355583191, 0.26454830169677734, -2.312725067138672, -0.5388805270195007, 0.1864539086818695, 0.9062358140945435, 0.04961663857102394, -0.9371511340141296, 1.8441640138626099, -0.5923710465431213, 0.9532828330993652, -0.6399512887001038, 0.5271240472793579, -0.3533214330673218, 1.4627141952514648, 0.8140898942947388, -0.23938776552677155, 0.889913022518158, 1.0903419256210327, -1.2940231561660767, 1.017159104347229, 0.9437099695205688, 1.1931672096252441, -0.18033263087272644, 0.23464876413345337, 0.7078279852867126, -0.6929899454116821, -0.3155335485935211, 0.14752191305160522, -1.5108305215835571, 0.1835194081068039, 0.4015372097492218, 1.5042545795440674, -0.7810849547386169, -0.2972106635570526, -0.38984784483909607, -0.5532094240188599, -0.34308525919914246, -1.7365866899490356, 0.8515950441360474, -0.07824499905109406, -2.15398907661438, -0.15372967720031738, -1.41368567943573, 0.44840237498283386, -0.6722376346588135, -0.06319689005613327, -0.06273061782121658, 0.6265144348144531, 0.12780793011188507, 0.8722822666168213, 0.6452915668487549, -0.9766624569892883, -0.1556396782398224, 0.6006209850311279, -1.0156139135360718, -0.22997644543647766, 0.9582175612449646, -0.8581337332725525, -0.9864963889122009, -1.9208219051361084, -0.13433527946472168, -0.05447189509868622, -0.02748580276966095, 0.5115960240364075, -0.5245344638824463, -0.9973596334457397, -0.25916236639022827, 0.0068678888492286205, 0.5017984509468079, 2.447031021118164, 1.9948943853378296, -0.3875981569290161, 0.9669729471206665, 0.2700866758823395, -0.5034968852996826, -0.3616115152835846, -0.5993353128433228, 0.2599080801010132, 2.0350542068481445, 0.34003791213035583, 0.7357794642448425, -0.8922290205955505, -1.0653594732284546, -0.01295690331608057, -1.8620738983154297, 1.4965745210647583, 0.12704816460609436, 1.9031059741973877, -0.12394088506698608, -1.1197620630264282, -0.7838609218597412, -1.1600230932235718, 0.7831295132637024, -0.21023330092430115, 0.12983813881874084, 0.2748013734817505, 1.3474613428115845, -0.3451049029827118, 1.0477086305618286, 0.6583913564682007, -0.5838912129402161, -0.3681235611438751, -0.8266986608505249, 1.800881028175354, 0.6528295278549194, -2.602311611175537, 1.6144788265228271, -1.585335373878479, -0.8144245743751526, -0.6151442527770996, -2.124164342880249, 1.7258901596069336, 0.2277839630842209, 0.6320870518684387, -0.6351157426834106, 0.7568826079368591, 1.3056286573410034, 0.8381404280662537, -0.07653781026601791, 0.11639724671840668, 1.3558993339538574, -0.045913152396678925, 0.6503022909164429, -0.7865550518035889, -1.4962033033370972, 0.8345885276794434, -0.8239274024963379, 0.42261746525764465, 1.1780524253845215, -1.3708373308181763, 0.8540470004081726, 0.46351781487464905, -0.3135058283805847, 0.04273523390293121, 0.4518795907497406, -0.9624018669128418, 0.40603095293045044, -0.9856678247451782, -0.395833820104599, 0.6194600462913513, -0.7136666774749756, -0.6582620143890381, 1.562018871307373, -0.6683512926101685, -0.4081851541996002, 0.3714701533317566, -0.5589548945426941, 0.4692041575908661, 1.144653081893921, 0.8277897238731384, -2.3503313064575195, -0.12848398089408875, -1.4801583290100098, 1.2000986337661743, -0.3632566034793854, -1.5689946413040161, -0.2363121211528778, 0.8707225322723389, -2.7214927673339844, 0.15476225316524506, 0.08072350174188614, -0.21887178719043732, 1.4409745931625366, -0.9833362102508545, 0.22939012944698334, 1.0853480100631714, -1.2731684446334839, -0.14583861827850342, 1.1151362657546997, 0.9533652663230896, -1.5211011171340942, 1.432107925415039, -0.4063926041126251, 2.4583446979522705, -0.09982508420944214, -1.3015551567077637, -1.5251497030258179, -1.1261929273605347, -0.20540982484817505, -1.3712354898452759, -0.0372975692152977, 0.7636803388595581, 1.1029771566390991, -1.0700304508209229, -0.9619395136833191, -0.12943078577518463, 0.7101166844367981, 1.153103232383728, -0.6299662590026855, -0.13565558195114136, 0.5475431084632874, -2.0534088611602783, 0.4293328523635864, 0.3331546485424042, 0.8701853156089783, 0.6241466403007507, -0.2465241551399231, 0.9453785419464111, 0.05664345622062683, -0.9833897352218628, 0.09923792630434036, -0.31749364733695984, -1.448323130607605, -0.5171196460723877, 0.6988071799278259, -0.8270959258079529, -0.6778404116630554, -0.5518313050270081, -0.8532445430755615, -0.7040605545043945, 0.008862834423780441, 0.06188477575778961, 0.9111056327819824, 0.01335524395108223, -1.868546962738037, 0.16324305534362793, -0.3048422038555145, 1.2074799537658691, 1.296147108078003, 0.6792605519294739, -1.9309790134429932, 0.7260159850120544, -0.8171393275260925, 1.1155166625976562, 0.17406097054481506, 0.0010802048491314054, -1.3822228908538818, 1.6457544565200806, -0.1632602959871292, 1.2541488409042358, 0.12006311863660812, -0.7244163751602173, 0.24476899206638336, -1.816011905670166, 1.610912799835205, 0.5498883724212646, 0.18374069035053253, 1.0976506471633911, -0.49763229489326477, -1.223007082939148, 0.33919474482536316, -0.4934505522251129, -0.19116130471229553, 0.6798227429389954, 0.49662894010543823, 0.6117246150970459, -0.2030773013830185, -1.2269833087921143, 0.24505193531513214, -0.20407117903232574, 1.6216408014297485, 1.91532564163208, -0.36566591262817383, 0.14256152510643005, -0.28361374139785767, 0.6337542533874512, -1.0259631872177124, -0.6292487978935242, 0.09956061840057373, -0.2888109087944031, 0.46092578768730164, -0.02764330804347992, 1.2655845880508423, -0.9780761003494263, 1.057700514793396, 1.4673094749450684, 0.29353049397468567, 1.7268084287643433, 0.7713574171066284, -0.9234017133712769, 0.6358239650726318, 0.8263963460922241, 0.1113845705986023, -0.536706805229187, -1.9260743856430054, 1.2489495277404785, -0.19844718277454376, 0.3283666670322418, -0.9086187481880188, 0.15389259159564972, 0.9421818852424622, 1.2680929899215698, 1.516785740852356, 0.8945726752281189, 0.33538541197776794, -0.5795643329620361, 0.6623511910438538, -0.1249818503856659, -0.09579131007194519, 1.0522937774658203, 1.6652448177337646, -0.5913771390914917, -1.9054275751113892, -0.5888690948486328, 0.6391168832778931, -0.48029017448425293, -0.6277110576629639, 0.21160532534122467, 0.42168423533439636, -0.2562536597251892, 0.5448521375656128, 0.8786514401435852, 0.14665694534778595, -0.5789745450019836, -1.6701723337173462, -1.4887946844100952, -1.6268635988235474, -0.2823752164840698, 0.452034056186676, 0.1144338920712471, -0.13262957334518433, 0.5445307493209839, -0.42337799072265625, 0.8970878720283508, -0.5332977175712585, -1.2600266933441162, -1.7262449264526367, -0.09677072614431381, 1.0727685689926147, 0.40415048599243164, 0.7428051233291626, 0.3459852635860443, -0.8779668807983398, -0.7694293260574341, -0.09645121544599533, 0.48558077216148376, 0.4455074071884155, 0.7501624822616577, 0.8005607724189758, -2.65326189994812, -0.2666609287261963, -0.11218073964118958, 1.8321441411972046, -0.6522964835166931, -0.8821427822113037, 0.1455696076154709, -0.30703049898147583, 0.110999695956707, 0.6115762591362, 1.703554630279541, -1.3405462503433228, 0.38215869665145874, -0.5022034049034119, -1.894054651260376, -0.6769058704376221, 1.178497314453125, 0.08937636017799377, 0.348067045211792, -0.22256405651569366, 0.8114750385284424, 0.49737387895584106, -0.2948615849018097, 1.0695490837097168, -0.3340519964694977, 0.6032297015190125, -0.09674008935689926, -0.07939057052135468, 0.48217877745628357, 0.7508091330528259, -0.033185042440891266, 1.2781753540039062, -0.1164441779255867, 0.32490479946136475, -0.10295973718166351, 0.033877503126859665, -1.059342861175537, -0.057605765759944916, -1.1448298692703247, -1.2687873840332031, 0.34974145889282227, 2.500767230987549, 0.1695757657289505, -0.11191366612911224, 0.25420624017715454, -0.8799614906311035, -0.6044836640357971, -0.024336323142051697, -1.2499275207519531, -1.2642320394515991, -0.6089242100715637, -2.457404613494873, -1.471315860748291, 0.17652113735675812, -0.6523264050483704, 0.38132575154304504, 0.31656840443611145, 1.0464004278182983, -0.19664856791496277, -1.6154717206954956, -0.3208019435405731, -0.16901490092277527, -0.27960965037345886, -1.6707266569137573, 1.0029295682907104, 0.04725084826350212, -0.2003026306629181, 1.7216986417770386, 0.7131722569465637, -0.3925663232803345, -0.622949481010437, -1.7275735139846802, 0.4137536585330963, 0.1505218744277954, 2.052307605743408, -0.8889592885971069, 1.4883939027786255, -1.9515280723571777, -1.7712516784667969, 0.42134130001068115, -0.746038556098938, -0.2579928934574127, 0.8917247653007507, 0.23731039464473724, -0.9938087463378906, 0.8646048903465271, 0.12655632197856903, 0.24980095028877258, -0.15033414959907532, 2.5134427547454834, -0.3913232684135437, -0.4759840965270996, 0.6081969141960144, 1.3695861101150513, 1.0113227367401123, -0.23817528784275055, -0.12515801191329956, 0.5350661873817444, 0.3433196544647217, -0.08404413610696793, -0.4903266131877899, 2.5646064281463623, 0.5469712018966675, 0.6388645768165588, 0.07665980607271194, 1.5118077993392944, -0.5735867619514465, 0.3936130106449127, 1.1227470636367798, -0.049897920340299606, -1.2201000452041626, -1.588700532913208, -0.4894353747367859, -0.9877184629440308, -0.2745976150035858, 0.14428991079330444, -0.9526770114898682, -0.14339962601661682, 1.7602893114089966, -1.1013352870941162, -0.163251593708992, -1.456388235092163, 0.5065657496452332, 1.2979130744934082, 0.2897239625453949, 0.5639234185218811, -0.18691667914390564, -1.6590055227279663, -0.0325106643140316, -0.3373218774795532, 0.1636299192905426, -1.1318421363830566, 0.0008175235707312822, -1.1578056812286377, 0.3484726846218109, -1.2561551332473755, 2.1816868782043457, 1.029003381729126, -1.1143823862075806, -1.578882098197937, 0.3940848410129547, -0.8542405962944031, 0.9117316007614136, 0.4542611837387085, -0.4516300857067108, -0.4003043472766876, -1.6711524724960327, -2.0675785541534424, -0.2335968166589737, -0.09845777601003647, 0.15593545138835907, -0.38012638688087463, 1.5028791427612305, 1.2868099212646484, -0.23118524253368378, -0.26242074370384216, -0.9781376123428345, -0.04251818358898163, -1.8027610778808594, 0.10346218198537827, 0.14884278178215027, 2.0568270683288574, -0.2515113353729248, -1.124457836151123, 1.7428991794586182, 0.5200095176696777, -2.794177293777466, -0.4954741597175598, -0.425339937210083, 1.3375821113586426, 0.8873385787010193, 0.5271705985069275, -8.903665730031207e-05, -0.8644589781761169, -1.0207065343856812, 0.9934836030006409, -1.715264916419983, 0.03369935601949692, -0.20237885415554047, 0.4620727002620697, 1.8084064722061157, 1.264406681060791, 0.7442417144775391, 0.20334799587726593, 0.6066207885742188, -0.07528305798768997, 0.13470853865146637, -1.5506631135940552, -1.3213757276535034, 0.458853542804718, 1.0788860321044922, 0.7698156833648682, -0.5721719264984131, 1.582647681236267, -0.20926828682422638, 1.0125128030776978, 0.5940359234809875, -0.8894449472427368, -0.7361917495727539, 0.8127419352531433, 0.16061623394489288, -1.5948296785354614, -0.31361186504364014, 0.12348160147666931, 0.45016205310821533, -0.16691528260707855, 0.5333155989646912, 0.009202389977872372, 1.9071112871170044, 0.2441040575504303, -0.8411648869514465, 0.8421733379364014, -0.13937664031982422, 0.15011732280254364, -0.3728290796279907, 0.3781697154045105, -1.7588965892791748, -1.252941608428955, -0.526951014995575, 2.3374295234680176, -0.0977819412946701, -0.7054556608200073, -0.7119215130805969, 0.2983773648738861, 0.7418656945228577, 2.549671173095703, -0.759289026260376, -0.8166070580482483, 0.5099955201148987, -0.3941382169723511, 0.4654967784881592, -0.5054634213447571, 1.4711788892745972, 0.3285995125770569, -0.8879099488258362, -2.0676729679107666, -1.1005713939666748, 0.9296364188194275, 1.3639322519302368, -0.3402705788612366, -0.3064137101173401, -2.691459894180298, 0.500968873500824, -0.11597388237714767, -0.0448896698653698, 1.4082202911376953, 0.9783543348312378, 1.0185602903366089, 0.17291873693466187, 1.9261631965637207, 0.2826324999332428, -2.1346614360809326, 0.07178130745887756, -0.06293776631355286, -0.844142496585846, -1.1858303546905518, 1.5639960765838623, 0.5475888252258301, 0.9364013075828552, 1.394734501838684, -0.05686301738023758, 0.4477366507053375, -1.4698352813720703, -1.0239534378051758, 1.8644927740097046, -2.0385584831237793, -2.1988649368286133, -1.3355042934417725, 0.5015720129013062, 0.8881092667579651, -0.788955569267273, 1.3767398595809937, 0.5497871041297913, 0.9371640682220459, 0.5322216153144836, -2.8389055728912354, 0.5840200781822205, 1.4988353252410889, 0.4220742881298065, 1.2328729629516602, -1.387449860572815, -0.5014136433601379, -0.20148636400699615, 0.16958191990852356, 1.3013548851013184, 1.5999935865402222, 0.010506161488592625, -0.9598778486251831, -0.6207160949707031, 0.1329505741596222, 1.8427115678787231, -2.382218360900879, -0.08610034734010696, -2.0793840885162354, 1.164392113685608, 1.181495189666748, -0.9795116186141968, -0.2528313100337982, -0.5476192235946655, 0.16953042149543762, 0.37283340096473694, -1.4796831607818604, -1.8326424360275269, 2.0578339099884033, 0.9399605393409729, 1.452133059501648, -0.5592098832130432, -0.12421567738056183, 0.8408616185188293, 0.13727033138275146, -2.1121394634246826, 0.24062468111515045, 0.18196411430835724, -0.8791183233261108, -0.21864958107471466, -0.6183729767799377, 0.32343125343322754, -1.105652928352356, -0.9661144614219666, 0.9749287366867065, 0.10763170570135117, 1.7903857231140137, -0.4031608998775482, -1.075556993484497, 0.13794513046741486, 1.6280364990234375, 2.3398303985595703, 1.6609461307525635, 0.8334309458732605, 1.4828912019729614, 1.4210783243179321, 1.0643272399902344, 0.09149184823036194, 1.6186292171478271, -1.4024147987365723, -0.7698475122451782, 1.8545411825180054, 1.2337082624435425, -0.7641618847846985, 1.6484737396240234, -0.92318195104599, 0.5069655179977417, -0.9480915069580078, 0.08572227507829666, 0.47778573632240295, 1.377428412437439, -0.5553008913993835, -0.9919483661651611, 0.8918206691741943, 1.5271718502044678, 0.24414744973182678, -0.03220982104539871, -1.1770073175430298, -0.02844362147152424, 0.5091170072555542, -0.18156416714191437, 1.7075424194335938, -0.05136044695973396, 0.5274986624717712, 0.7951171398162842, -0.12278951704502106, 0.6982341408729553, -1.3687870502471924, 0.37764427065849304, -0.750607967376709, 2.1341683864593506, -0.5495700240135193, -1.426481008529663, 1.0895715951919556, -2.4935858249664307, 0.1406751126050949, -0.2600799798965454, -0.21476446092128754, -0.5275382399559021, 0.23877373337745667, 0.5020975470542908, -1.1909908056259155, -0.5797685384750366, -0.0007903701043687761, -0.8786298036575317, 1.242503046989441, 1.3198106288909912, -0.23923543095588684, -0.6357178092002869, -0.016301793977618217, 1.9747300148010254, -0.37885379791259766, 0.6900078654289246, -0.7046113014221191, -0.7113752365112305, -0.647565484046936, -0.6770967245101929, -1.3286501169204712, 1.7783589363098145, 0.017463665455579758, 0.7439996004104614, -0.2790515422821045, 2.6458940505981445, 1.347276210784912, -0.7581734657287598, 0.7082127928733826, 0.16696424782276154, 0.8375544548034668, 0.6182700991630554, -3.376373052597046, -0.16719657182693481, -1.739076018333435, -0.11493951827287674, 0.4158307909965515, 0.4164673388004303, 0.5182874798774719, -1.4708095788955688, 0.48301130533218384, 0.16140441596508026, 0.335772305727005, -0.5544497966766357, 0.8184113502502441, 0.2570919990539551, 0.11122142523527145, 1.4112216234207153, -3.1444125175476074, 0.992519736289978, -0.8922148942947388, -0.011992299929261208, 0.10063378512859344, -0.6096907258033752, -2.5918798446655273, -0.1469387412071228, -1.1124932765960693, 0.10601039975881577, -0.5229986310005188, 0.4704074561595917, -1.8323312997817993, -2.4429702758789062, 0.6551702618598938, -0.29187092185020447, 1.7374227046966553, 1.2213466167449951, -0.5053969025611877, 1.5671379566192627, -0.8450004458427429, 0.10481062531471252, -1.0599453449249268, 0.7879227995872498, 0.03571438789367676, 0.5925143957138062, 1.7320111989974976, 0.3767987787723541, -0.26967373490333557, 0.6459836959838867, -0.43629783391952515, -0.4330555498600006, 1.6474857330322266, -1.2560431957244873, 0.018702102825045586, 0.9177513122558594, -1.7308015823364258, 0.2582091987133026, 0.416411817073822, 2.581754446029663, 0.17086222767829895, 0.5985826849937439, -0.7995717525482178, -0.7387635111808777, -2.1025145053863525, -0.12475670874118805, 0.774823784828186, 1.0649218559265137, 0.42960649728775024, -0.8005449771881104, -1.0769929885864258, -0.1917065680027008, -0.6561276912689209, -0.4318327009677887, -0.06998147070407867, -0.2288043200969696, -0.7217137217521667, -2.1208624839782715, 0.08903024345636368, 0.0204799622297287, -0.5057572722434998, 1.1193978786468506, -0.6076812148094177, -1.0071057081222534, 0.6136565208435059, 0.34688353538513184, -1.0419830083847046, 0.10795913636684418, 3.244448661804199, 0.9433088898658752, 0.19799764454364777, 1.057651162147522, 0.9535952806472778, 0.9270176291465759, 2.1578209400177, -1.1984798908233643, 0.33225080370903015, 0.28423652052879333, 0.17754562199115753, 0.7588433623313904, -0.7238733172416687, -1.8567825555801392, -2.4359822273254395, 0.5884601473808289, -0.48772162199020386, 0.25429341197013855, 2.4258339405059814, 0.4297519028186798, -0.881767213344574, 0.1923457831144333, 0.8750409483909607, 0.7241321206092834, 0.21860988438129425, 0.07821831852197647, 0.16249966621398926, 0.07368617504835129, -0.8484506607055664, 0.5046682953834534, 1.2586997747421265, 1.152268409729004, 0.5749567747116089, 0.6834666728973389, -0.7758159041404724, -0.28826263546943665, -1.1676441431045532, 0.2846190631389618, 1.005667805671692, 0.09605064988136292, 0.028651578351855278, -0.6748529076576233, -1.525051474571228, 0.1612122654914856, -1.2748531103134155, -0.38948726654052734, 1.045772910118103, -1.8857738971710205, 1.029413104057312, -0.763197124004364, -0.7939722537994385, 0.2953738868236542, 0.0978204682469368, 0.7731595039367676, -1.8865102529525757, 2.4481189250946045, 0.5104820728302002, -1.1309831142425537, -0.1799546629190445, -0.294023722410202, 0.7615494728088379, 0.04469091445207596, 0.2265840321779251, 1.2338337898254395, 0.03187141194939613, 0.012435850687325, -0.0406377874314785, -1.0610815286636353, -0.7990363836288452, -1.1322877407073975, -0.4139857888221741, -0.3269117772579193, -1.0833101272583008, 1.8993579149246216, 1.7242693901062012, -0.9290274381637573, -1.5558595657348633, -0.5935162305831909, 0.20999595522880554, 0.41761815547943115, -0.5666552782058716, 1.7254430055618286, 0.5634161829948425, -2.4718360900878906, 0.13304540514945984, 1.0210423469543457, -0.6694743037223816, 0.6193877458572388, 1.163101077079773, -1.018306016921997, 0.5212432742118835, 1.2498291730880737, 0.8266430497169495, -0.36706098914146423, 0.1326274871826172, 1.300756812095642, -0.4428197741508484, 0.5922386646270752, 0.6636636853218079, -0.7545831799507141, -1.3752501010894775, -0.5774328112602234, -1.2775505781173706, -1.6425137519836426, -0.6084138751029968, 2.08371639251709, -1.3568271398544312, -0.017140761017799377, 1.5760971307754517, -0.5856401920318604, -0.04937044531106949, 0.1607433408498764, -0.707129180431366, -0.0688781812787056, -0.4380635917186737, 0.5074251294136047, 2.79525089263916, 2.3505616188049316, -1.074387550354004, 1.2652252912521362, 0.8517245650291443, 0.3499809503555298, -0.6152763962745667, -0.7329707741737366, -1.5848122835159302, 0.5865081548690796, -1.3173272609710693, -1.009800672531128, 1.5651828050613403, -1.2556347846984863, 0.6605116128921509, -1.4044300317764282, 0.594639778137207, 0.4821805953979492, 1.6128474473953247, -0.34208348393440247, -0.0753272995352745, 0.9907260537147522, -0.527640700340271, 0.540116548538208, -1.5839110612869263, -1.7415093183517456, 0.9613848328590393, 0.3502761721611023, 1.2617719173431396, 2.0567147731781006, 0.859798789024353, -2.8022336959838867, 1.0584819316864014, 0.09504958242177963, -0.3679662048816681, 0.20876090228557587, 0.7344779372215271, 1.0032248497009277, 2.7474706172943115, -0.8197070956230164, -0.018114866688847542, 1.6741865873336792, 0.37594324350357056, 1.09107506275177, 1.0929096937179565, 0.1295587718486786, 0.11949094384908676, -1.1738653182983398, -0.15791378915309906, 0.789107620716095, -1.4892743825912476, -0.910643458366394, 0.24461664259433746, 0.8964241147041321, -0.010411434806883335, 0.016518063843250275, -0.1785014271736145, 0.5649964213371277, -2.0415267944335938, -0.1943889856338501, -0.7737498879432678, -0.24617400765419006, -1.0222461223602295, -1.3368748426437378, -1.1247740983963013, -2.275059223175049, 0.08087095618247986, -0.7916865944862366, 0.36606645584106445, -0.33693987131118774, -0.768712043762207, 0.891413688659668, 1.066592812538147, 0.1383601874113083, -0.626177966594696, 1.233461856842041, 0.5922091603279114, -0.48011547327041626, -1.6593321561813354, 0.1893167793750763, -0.029568029567599297, -1.3397175073623657, -0.7663547396659851, -0.5054929256439209, 0.47833266854286194, -0.7220109701156616, -0.5440124869346619, 1.0580555200576782, -0.04009683057665825, -1.3245388269424438, -1.4996601343154907, 0.44960975646972656, -0.04294456169009209, 0.03247491270303726, -0.7096988558769226, 1.5335386991500854, -0.5384576916694641, -0.20957358181476593, -0.6077364683151245, -0.9686856865882874, 1.548959493637085, -0.35813605785369873, 0.2579073905944824, -0.3378271460533142, -0.4542374312877655, -0.4372817277908325, 0.6329439878463745, -0.24756185710430145, -0.03788308799266815, 0.5540109276771545, -1.528286099433899, 1.8682613372802734, 0.9346840381622314, -0.7119626998901367, -0.45457062125205994, -0.9894464612007141, -0.419381707906723, 1.0411978960037231, -0.2966029644012451, -1.2787922620773315, 0.14066720008850098, -1.8364880084991455, 0.18943174183368683, 0.009449275210499763, 1.5935273170471191, -1.390634536743164, -0.046412449330091476, 1.9673417806625366, -0.9444578886032104, 0.4428768455982208, -0.4368968904018402, -0.8795311450958252, -0.04788763448596001, -0.4822620451450348, -0.13548307120800018, 1.127883791923523, 0.539492666721344, 0.2869671881198883, 0.5844150185585022, 0.45693960785865784, 1.1300796270370483, -0.11986828595399857, -0.3632363975048065, -0.7382030487060547, -0.6864420771598816, 1.4787894487380981, -0.5990409255027771, 0.16576577723026276, -0.7074102759361267, 0.1939084678888321, 0.7395896315574646, 1.8162012100219727, -0.0008719244506210089, -1.0651711225509644, 0.6333182454109192, -0.8377040028572083, -1.1060987710952759, -0.8170157670974731, 0.052615661174058914, -1.131645917892456, 0.8232970833778381, 0.6486843824386597, 0.5920698046684265, 0.8326064348220825, 1.9951471090316772, 0.7578713893890381, -1.6551051139831543, -0.6207992434501648, 1.6971435546875, -2.2170403003692627, 0.2752850353717804, 1.903991937637329, 0.17720144987106323, 0.8227920532226562, 1.0426406860351562, -1.1418386697769165, 0.31901660561561584, -0.5778357982635498, 0.6406446099281311, 1.0363128185272217, -2.0316717624664307, 0.01842830702662468], [-0.47324615716934204, 0.24504075944423676, 0.6372461318969727, -0.5629396438598633, 2.1734554767608643, -0.32718780636787415, -0.31294503808021545, 0.4733823835849762, -0.03787032887339592, 0.11855384707450867, -1.3689839839935303, -0.8224349617958069, -0.27821528911590576, -0.21022601425647736, -2.1042847633361816, -0.541039228439331, -0.9777728915214539, -0.8166249990463257, 0.5832358598709106, 0.28370970487594604, 0.23560303449630737, -0.7918380498886108, -0.361690491437912, -1.3916957378387451, 0.061004698276519775, 1.7479329109191895, 0.5287851691246033, 3.020399808883667, 2.676223039627075, -0.5582095980644226, 0.2662523686885834, -0.10556779056787491, -0.9791647791862488, -0.529721200466156, -0.6272051930427551, -1.0852198600769043, 0.40204495191574097, -1.7426323890686035, 1.7165312767028809, -0.3256931006908417, -0.6336883902549744, 1.2870854139328003, -0.36333826184272766, 0.2664085030555725, -1.1653658151626587, 0.7331434488296509, -0.8615776896476746, 1.3994139432907104, -1.0570555925369263, -1.7029039859771729, -0.15393660962581635, -1.2960705757141113, -2.252852201461792, 0.5532003045082092, -0.5371352434158325, 1.0173429250717163, -0.6512819528579712, -0.3509849011898041, 0.6617702841758728, 0.5877033472061157, -0.7993980646133423, -0.9567812085151672, 0.8673948049545288, -2.1581859588623047, -0.00911880936473608, -0.3467143177986145, -0.028695233166217804, -0.1647091805934906, -1.6205625534057617, -0.8128501772880554, 0.7477802634239197, 0.8718754649162292, -0.13178907334804535, 1.1627895832061768, -0.726854681968689, 1.400830864906311, 0.6200476288795471, -1.6443268060684204, -1.1121184825897217, -0.282980352640152, 2.825310230255127, 0.14603377878665924, 0.3121701776981354, -0.03623460233211517, -0.46773838996887207, -0.1527933031320572, 0.2604467272758484, -0.42699941992759705, -0.8493760824203491, -1.1641114950180054, 1.2207648754119873, 1.1099042892456055, 0.3610153794288635, 1.5953844785690308, -0.37564679980278015, -0.2238110452890396, 0.8156929016113281, -0.85914546251297, 0.771660327911377, 0.6983388066291809, -0.5925909876823425, 1.222977876663208, 1.5240315198898315, 0.022587988525629044, -0.25586196780204773, -2.660221815109253, -0.23605488240718842, -0.0407915823161602, 0.3021475374698639, -2.612748622894287, 0.8152597546577454, -0.40448272228240967, 0.5688485503196716, 1.3129041194915771, 0.8819817304611206, -0.49880993366241455, -0.16174641251564026, -0.47017592191696167, -1.8953161239624023, 0.29937082529067993, 0.4014366865158081, -0.9580591320991516, 1.3017958402633667, -1.126674771308899, 0.5374183058738708, 1.8132543563842773, -1.8979597091674805, 0.17142003774642944, 0.3252803087234497, -0.6628963351249695, -0.24720196425914764, -1.2603397369384766, 1.0735446214675903, -0.2091774195432663, 0.8705701231956482, 0.45427006483078003, -1.3961472511291504, -1.3126447200775146, 0.9583479762077332, -1.946056842803955, -0.9561469554901123, -0.6077266931533813, -0.13982318341732025, 2.0073437690734863, -0.3147863447666168, -0.26230618357658386, 0.3958110511302948, -0.3488745391368866, -0.12600575387477875, 0.9252557754516602, -0.6958990693092346, -0.003543326398357749, 0.5288500785827637, 0.7431691288948059, -1.3280454874038696, 1.1216344833374023, 0.6009361147880554, 0.8083206415176392, -0.07398856431245804, 0.01670830510556698, -0.6892577409744263, -2.8849642276763916, 1.5442023277282715, 1.0966845750808716, 1.5837568044662476, -0.3207201063632965, 0.42098021507263184, -0.16709819436073303, -0.4769018590450287, -0.17014378309249878, -1.3574395179748535, -0.5282725095748901, -1.7045741081237793, 1.1727402210235596, 0.960732638835907, -1.9180516004562378, -0.6350001692771912, -0.7762055993080139, -0.5771504640579224, 1.274977207183838, -0.5429076552391052, 1.823686122894287, 0.30110788345336914, 0.13338032364845276, -1.61603844165802, 0.09757554531097412, 0.4482339322566986, 0.1827976107597351, 2.0820066928863525, -0.9374790191650391, -0.5241536498069763, -2.2656912803649902, -0.16063350439071655, 0.4840615391731262, 0.8178518414497375, 0.6313183307647705, -0.44587448239326477, -0.49397966265678406, 0.27006393671035767, -1.3485466241836548, 0.3881570100784302, -0.010791494511067867, 1.0852651596069336, 0.2451263815164566, -0.7433463335037231, -0.10952345281839371, 0.15018722414970398, -0.2504589557647705, 1.8095386028289795, -1.2218585014343262, 0.5112580060958862, -0.5046663284301758, 0.9272847175598145, 0.17723946273326874, -0.19749504327774048, 0.9775159955024719, 0.7360678315162659, -0.5581793189048767, -0.5354632139205933, 0.49626076221466064, 1.6670093536376953, -1.6100374460220337, -0.20880822837352753, -0.7695369124412537, 0.4894495904445648, 0.1365002542734146, -0.8917623162269592, -0.5068514347076416, 0.748892605304718, -0.5602350831031799, -0.47042587399482727, -1.351625680923462, 0.4740193784236908, -0.5732108950614929, -0.08714760094881058, -1.9809995889663696, 0.26285719871520996, -1.068642258644104, 1.2841577529907227, -1.342384696006775, 0.2586333751678467, -0.8046877980232239, -0.3373984098434448, -1.100315809249878, 0.7203510999679565, 0.2665470540523529, 1.6431742906570435, -0.6635724306106567, 0.2608792185783386, 0.1891423612833023, -0.32059210538864136, 1.1873639822006226, 0.1380060911178589, -0.10836756974458694, -0.7542662620544434, 0.4141309857368469, 0.5268086194992065, 0.4556403160095215, 0.3034978210926056, -0.39525383710861206, -1.0506670475006104, -0.578066885471344, 0.11570879071950912, -0.40448471903800964, 0.7320573925971985, -0.8417706489562988, -0.3697054088115692, 1.194550633430481, -0.2887537479400635, 0.42250752449035645, 0.6923671364784241, -0.6327247023582458, -0.18706053495407104, -0.4231050908565521, 1.3584169149398804, 0.5105910897254944, -0.7061873078346252, 0.5018646717071533, 0.3494178354740143, 0.742670476436615, -2.089804172515869, 0.9791284799575806, -1.6667520999908447, 0.5245078802108765, 0.4324073791503906, 0.34946900606155396, -0.6489054560661316, -0.6821127533912659, 0.6826233863830566, -0.9766039848327637, -0.20322152972221375, -0.9472658038139343, -1.214667797088623, -0.14142678678035736, -0.22464238107204437, 0.5114238858222961, -0.20865948498249054, 0.13732315599918365, -0.033051054924726486, 0.3481878638267517, -0.24157053232192993, -1.9773194789886475, 0.7195454239845276, 0.8318431377410889, 0.7349891662597656, -1.0141345262527466, -0.6686508059501648, 1.0239485502243042, 0.07942191511392593, 0.20067960023880005, -0.17919746041297913, 0.16484437882900238, -2.159579038619995, -0.3860703706741333, 1.5929139852523804, 1.1775003671646118, -0.4387795627117157, -0.36482349038124084, 0.5542201995849609, -0.4075160324573517, -0.07266152650117874, 1.1148533821105957, -1.4628658294677734, -0.2174304574728012, 1.0390944480895996, -0.2133491337299347, 0.8277953267097473, -1.0434327125549316, 1.2672003507614136, -0.07172248512506485, 0.13625119626522064, 1.4384307861328125, 0.20468100905418396, 1.219907522201538, 0.07682391256093979, -1.4749215841293335, 1.7148261070251465, 0.3963910937309265, -0.021732930094003677, -0.08875330537557602, -0.32275882363319397, -0.0093027139082551, -1.3596539497375488, -1.2201377153396606, -0.43218278884887695, 1.0284972190856934, 0.17527076601982117, -0.3810499310493469, 1.7922462224960327, 0.25310733914375305, 0.6296570301055908, -0.221378356218338, -1.7775768041610718, 0.49354609847068787, -0.6862542629241943, -0.7783607840538025, 0.0650404766201973, 1.3945484161376953, -0.7297154068946838, -0.24702025949954987, -0.5606324076652527, -0.3680671155452728, 0.6683206558227539, 0.6362619996070862, -0.7426506280899048, 0.15293574333190918, 0.5679618120193481, 0.8185796737670898, 0.2685873508453369, 0.22702684998512268, -1.2133115530014038, 0.4598594307899475, 2.1324806213378906, -0.7861559391021729, 1.2244869470596313, 1.2208839654922485, -0.0012599964393302798, 0.44146233797073364, -1.6257274150848389, 1.902174711227417, 0.6959406137466431, -0.3275391459465027, -0.5702685117721558, 0.2381817251443863, -0.5049576759338379, -0.9196372032165527, 1.6581923961639404, -0.615632176399231, -2.407860517501831, -0.5389862060546875, -0.3952403664588928, 1.2974934577941895, 0.8159996271133423, 1.0847474336624146, -1.1572726964950562, -0.8134561777114868, 0.7785183787345886, 0.21925956010818481, -1.5377600193023682, -1.5532606840133667, 0.8399366736412048, -0.05961153283715248, -1.3147801160812378, 0.5557023882865906, -2.227343797683716, 1.9113081693649292, 0.26974406838417053, 0.31778573989868164, 0.49669092893600464, -0.8898216485977173, -0.30946090817451477, -0.8667150735855103, 0.311481237411499, -1.7138465642929077, 1.4492461681365967, 3.2574799060821533, 1.1758278608322144, 0.8546092510223389, 0.6327483654022217, -0.4835813045501709, 1.0386812686920166, -1.1900246143341064, -0.3490079343318939, 1.8898053169250488, 0.43857619166374207, -0.28349801898002625, -0.09928730130195618, -2.3592331409454346, 0.06072094663977623, -0.1206912249326706, 0.391406774520874, -0.010094402357935905, -0.26167163252830505, -0.2740824222564697, 0.6180304884910583, 1.9385406970977783, -0.7274677157402039, 0.6103304624557495, 1.9024323225021362, 0.7791069149971008, 0.6215730905532837, 0.04484943300485611, 0.6456714868545532, 0.13688331842422485, 0.6441977024078369, -1.2568846940994263, -0.2770552337169647, -0.1857982575893402, 0.7162321209907532, 0.4460979104042053, 0.2510759234428406, 0.9226458072662354, 0.0992806926369667, 0.32127049565315247, 0.27871912717819214, -0.08144480735063553, 1.644281268119812, 0.46753114461898804, 0.9001454710960388, 1.366411566734314, -0.6865777969360352, 0.19524817168712616, -2.215365171432495, -0.5938156843185425, -2.169583320617676, -2.644695281982422, 1.3891527652740479, 0.9536904096603394, -1.2442718744277954, -0.3311854600906372, 0.4070555865764618, -0.40243446826934814, 0.4096475839614868, -0.9563648700714111, 1.2042741775512695, -1.2736785411834717, -1.382935881614685, -1.3559216260910034, -1.1092487573623657, -0.5834128856658936, 1.1508527994155884, -0.8225204348564148, 1.3171323537826538, -0.2384747415781021, -1.031996250152588, 0.4250571131706238, 0.3583624064922333, 0.19933180510997772, 0.5304847359657288, 0.5943119525909424, -0.6889209747314453, 0.7430741786956787, -0.7767822742462158, -0.8645097017288208, 0.8007364869117737, -0.11013875156641006, 1.1132564544677734, -0.35952258110046387, 0.4773384928703308, -0.14500898122787476, -1.1820403337478638, 0.5177360773086548, 1.0867995023727417, 0.779405415058136, -0.20321665704250336, -0.5604244470596313, -0.014771221205592155, -0.2860265076160431, 0.5648967623710632, -0.22866621613502502, -0.9970694184303284, -0.5454545021057129, -0.43658187985420227, -1.7117023468017578, -0.34118419885635376, -0.12464984506368637, 0.168363556265831, 1.1337413787841797, -0.0957828015089035, -0.11940517276525497, 0.7788631916046143, -1.72760009765625, -1.4562653303146362, 0.17221665382385254, 0.019371353089809418, -0.3405963182449341, -0.023219697177410126, -1.4826810359954834, -0.18213407695293427, -1.3791924715042114, -0.9991166591644287, -1.3303248882293701, 0.3786148726940155, -0.41079357266426086, 0.5339653491973877, 1.0766890048980713, 1.2714524269104004, 0.2161477506160736, -0.440365731716156, -0.9656835198402405, 0.9652654528617859, 0.6110551357269287, -0.20120950043201447, -0.19871307909488678, -1.7825852632522583, -0.08423397690057755, 0.27367791533470154, -2.0197248458862305, -0.5190528035163879, 0.25177618861198425, -0.5790706872940063, -0.8097650408744812, -1.0702437162399292, -0.4422600567340851, -0.4763539433479309, 0.7280227541923523, 0.41860517859458923, -1.6023688316345215, -1.1529449224472046, -0.2836444675922394, 1.9936565160751343, 0.6257508993148804, -1.4330190420150757, 0.5904620289802551, 1.7738181352615356, 1.6879277229309082, -0.8321195840835571, -0.6773459911346436, -0.5587412714958191, 0.46560242772102356, 0.7774304151535034, 1.0611708164215088, 0.4324527978897095, 0.30408769845962524, 0.2707895040512085, -0.11665920168161392, 0.20900613069534302, -0.22604084014892578, -2.3305060863494873, -0.3549993336200714, 0.6947057247161865, -0.14954280853271484, -0.7842440009117126, 0.31014129519462585, 0.4036848545074463, -0.007751236204057932, 1.0617653131484985, 0.20408441126346588, -0.27977877855300903, -1.127341389656067, 0.9941096901893616, 1.7693121433258057, -0.7675529718399048, -0.16973961889743805, 0.4323890507221222, 1.8395135402679443, -2.6051454544067383, 0.6675324440002441, 0.8425177931785583, -1.1461894512176514, 0.5940990447998047, -0.2812149226665497, -0.6807257533073425, 0.8671762943267822, 0.8182909488677979, 0.3936771750450134, -0.6926145553588867, -0.5859261751174927, 0.6315349340438843, -0.33104798197746277, 0.9321116209030151, 0.3650103509426117, 0.597884476184845, 0.7097492814064026, -0.24471327662467957, -0.5903480052947998, 0.5426974296569824, 0.984457790851593, 0.5237823724746704, 1.0129305124282837, -0.8593330979347229, 0.25949421525001526, 0.3613984286785126, 0.7344313859939575, -1.0624665021896362, 0.06307446211576462, -0.13791994750499725, -0.6669628024101257, 0.22518698871135712, -0.05513438209891319, 0.3283192813396454, -2.441406488418579, 0.6889968514442444, 0.21172291040420532, 0.6763247847557068, 0.4621792733669281, 0.25239068269729614, -0.0769130289554596, 0.014462083578109741, -0.6996081471443176, 1.975440263748169, 1.2477023601531982, 0.8129745125770569, -0.2224912792444229, -0.8870156407356262, -1.6648379564285278, -0.49852943420410156, 2.6258747577667236, -0.6067415475845337, -1.837317705154419, -0.5671564936637878, -1.9258754253387451, -1.4100102186203003, 0.10037059336900711, -1.5052545070648193, -1.0748255252838135, -0.35813361406326294, -1.7168774604797363, -0.8399543762207031, 0.15473361313343048, -0.1211952418088913, -1.994762659072876, -0.2523939311504364, 0.3554479777812958, 1.1200963258743286, -0.008022853173315525, -1.1721891164779663, 0.6361673474311829, -1.9928396940231323, 0.26499220728874207, -0.6411365866661072, -1.424139380455017, 1.0769233703613281, -0.6208529472351074, -0.11424808949232101, 0.08622987568378448, -0.6790702939033508, 1.370849370956421, 1.0702072381973267, 0.31738579273223877, -0.5685996413230896, 0.6355162262916565, 0.3509698510169983, -0.47871825098991394, -0.5975624918937683, 0.9623889327049255, -2.312173843383789, -0.38064950704574585, 1.301236629486084, 1.834080696105957, -1.5380711555480957, 1.2207475900650024, -1.607816457748413, -0.41707780957221985, 0.9098396301269531, -1.0306990146636963, 0.46468880772590637, -0.8631423115730286, 0.33666497468948364, -0.11351799219846725, 0.41029128432273865, 2.0915069580078125, 1.6383743286132812, 1.0779484510421753, -1.3773859739303589, -1.1811213493347168, 1.451285719871521, -2.2486038208007812, 0.6381502747535706, 2.6642005443573, -0.7837300300598145, 0.21123535931110382, 2.1782069206237793, 0.06495596468448639, 1.0261785984039307, -0.07015176862478256, -1.2163466215133667, 0.93808913230896, 0.32143908739089966, 0.3016892373561859, -0.7268018126487732, -1.279948353767395, 0.4361230134963989, -2.700364112854004, 0.19980649650096893, 0.5085121393203735, -0.4226579964160919, -1.3094812631607056, 0.31468334794044495, -0.022790901362895966, 1.0729548931121826, -0.5745263695716858, 0.04071497917175293, -0.1908642202615738, 0.4220584034919739, -2.623344898223877, -1.3417788743972778, -0.22240327298641205, 1.0955009460449219, -0.4202009439468384, -1.7238084077835083, 1.9131734371185303, 1.5035423040390015, -0.7993165850639343, 2.6271839141845703, -0.11717867851257324, 0.8769388198852539, -0.9947180151939392, -0.06640926003456116, 0.3584415018558502, 0.3198792338371277, 0.3170228600502014, -0.749661922454834, -0.41028058528900146, -0.4867998957633972, -0.31910696625709534, 1.7547210454940796, -0.012761136516928673, 0.8242805600166321, 0.083429254591465, -0.710178554058075, 1.8742129802703857, 1.859676480293274, -0.2548494338989258, 0.15618236362934113, 2.042752265930176, -0.07502839714288712, 0.26846975088119507, 0.6851041913032532, -0.08354998379945755, -1.0568181276321411, 0.07844004780054092, 0.4616911709308624, 2.1540627479553223, -0.016675429418683052, 0.4563906490802765, 0.9653336405754089, 0.2101825326681137, -0.6453760266304016, 2.1273882389068604, 0.5186284780502319, 1.1861215829849243, -1.1171879768371582, 1.4419751167297363, 0.9907180070877075, 3.2405831813812256, -1.7695133686065674, 0.17143450677394867, 2.385617971420288, 0.7920109033584595, -1.8512780666351318, -0.46360543370246887, -0.3308388888835907, 0.7825649380683899, 1.1540446281433105, 1.6130887269973755, 0.9234985113143921, -0.1564887911081314, -0.2610306441783905, -0.2885543406009674, -0.6175037026405334, -0.5788180828094482, 0.5745135545730591, 0.25462275743484497, -0.6890841126441956, 1.1167101860046387, 1.341543197631836, 1.1666642427444458, -1.5390479564666748, -0.5548768043518066, 0.4608907699584961, -1.543148398399353, -0.18866804242134094, -1.3923983573913574, -1.0266494750976562, 0.1948341280221939, 0.3604312837123871, -1.2769138813018799, 0.3832783102989197, -0.35565486550331116, -0.8504864573478699, 0.8950174450874329, -0.3555138409137726, -0.8331775069236755, 0.14295406639575958, -1.0583510398864746, 0.34632188081741333, -0.5099215507507324, -0.3484855890274048, -2.1517813205718994, -0.22373735904693604, -0.0035760339815169573, -1.9770599603652954, 0.9374858736991882, 0.4047725796699524, 0.35833683609962463, 0.3025534451007843, 0.8869684338569641, -0.5518481731414795, -0.052106983959674835, 0.15387238562107086, 1.3167314529418945, 0.842232346534729, -1.1325552463531494, -0.538184404373169, 0.8052988052368164, 0.032896846532821655, -0.08329730480909348, -0.6314806342124939, -0.8612285256385803, -1.0608446598052979, 0.5441895127296448, 0.6503093838691711, 1.5002464056015015, -0.8957933187484741, -0.20729339122772217, -2.52695369720459, -2.1369874477386475, 0.8159461617469788, -0.7368955612182617, -0.6478642821311951, -0.5185278058052063, 1.6687315702438354, -0.16302721202373505, -1.2985904216766357, 1.8873282670974731, 0.2691081464290619, 0.6216808557510376, 1.5933223962783813, -0.9164083003997803, 0.09209494292736053, 1.6331897974014282, -0.05831047147512436, -0.7915928363800049, 1.395692229270935, -0.6091349124908447, -0.2827281653881073, 0.27204516530036926, -0.18353839218616486, -0.7153234481811523, 0.39069488644599915, 1.1077406406402588, 0.3621155917644501, 0.9030588269233704, -0.1989945024251938, 0.6593272686004639, 0.3036547601222992, 0.7899812459945679, 0.6895772814750671, -0.08286290615797043, -1.1152805089950562, 1.8648216724395752, -0.37388426065444946, 0.2136799395084381, -0.2065936177968979, 1.3746715784072876, -1.1488687992095947, -0.055839065462350845, 0.14162610471248627, -1.3109748363494873, -0.3172461688518524, 0.7944817543029785, 0.6313091516494751, -1.6930253505706787, -1.2259886264801025, -0.9536257386207581, -1.0273271799087524, 2.375307559967041, 0.7285178899765015, 0.18531601130962372, -0.9728202819824219, -0.3385392725467682, 0.676807701587677, 2.070610284805298, 0.6152306795120239, -0.7252219915390015, -1.1072505712509155, 3.0822184085845947, 0.99319988489151, 0.44234922528266907, 0.25274527072906494, -0.5075295567512512, 0.8972060680389404, -0.3047943711280823, -0.8133560419082642, 0.042365118861198425, -1.8181740045547485, 0.5509679913520813, 1.407167673110962, 1.2365689277648926, -1.4345390796661377, -1.5133883953094482, -0.7440100908279419, -2.377018690109253, 0.5327603816986084, 0.7280781269073486, 0.6044307351112366, -0.5743938088417053, -0.15696130692958832, -0.37488535046577454, 0.41801196336746216, 1.540221095085144, 1.8541136980056763, 0.44319984316825867, 0.9743925929069519, -0.7611770033836365, 0.498191773891449, 0.40320080518722534, -0.13556604087352753, -0.41346147656440735, 0.4404195249080658, -0.29891470074653625, -1.5180144309997559, -0.5759688019752502, -0.44727441668510437, -1.4291024208068848, -1.6587226390838623, -0.7435058355331421, -1.2864307165145874, -0.12854160368442535, -0.5774962902069092, 1.903892159461975, 0.5413126945495605, 1.3898793458938599, -0.2907496392726898, -0.41455528140068054, 1.1797982454299927, 2.51212477684021, -0.4194955825805664, 0.6895633339881897, 0.28658756613731384, -1.44035005569458, 0.4658842086791992, 1.1415551900863647, -0.09065498411655426, -0.10252497345209122, 0.8796976208686829, -1.177839994430542, -1.752675175666809, 0.6687636375427246, 0.4515646994113922, -0.35650181770324707, 0.7017508745193481, -2.3968124389648438, -1.7542532682418823, 0.24237163364887238, 0.9310365915298462, 0.8572066426277161, 0.8637111186981201, 1.1170251369476318, 1.6886836290359497, 0.6475275158882141, 2.2638673782348633, 1.3466242551803589, 1.938439130783081, 0.38686391711235046, 0.7035261988639832, 0.09257350862026215, 1.2181881666183472, -1.0455753803253174, -1.4153765439987183, -1.5046303272247314, 0.28563159704208374, 0.6390761733055115, -0.38733604550361633, 0.9508570432662964, -1.3608518838882446, 0.6522745490074158, 0.38421860337257385, 0.15966515243053436, 0.5940800309181213, -0.8985939025878906, 0.7858738303184509, -0.4164813160896301, -1.521016001701355, 1.0051188468933105, 0.19688071310520172, -0.06548422574996948, 0.8561607599258423, 0.3121735751628876, 0.12492503225803375, 0.34936389327049255, -0.6728104948997498, 1.044474720954895, -0.28880414366722107, -0.7853613495826721, 1.1120309829711914, 0.23417913913726807, -1.280903697013855, -0.6740223169326782, 0.7019776105880737, -1.4511092901229858, -0.9849467277526855, -0.514506995677948, 0.025380462408065796, -1.641062617301941, 0.12050008773803711, 0.9028627872467041, -1.3023650646209717, 0.9044103622436523, -0.08039024472236633, 0.1280280202627182, -3.199876546859741, -1.3165363073349, -1.2491867542266846, 0.18993419408798218, 1.5867409706115723, 0.4941553771495819, 0.6383179426193237, -0.2627415955066681, 1.3824008703231812, -0.9861032366752625, -0.7748856544494629, 0.5237718820571899, 0.016154395416378975, 1.654131293296814, 1.5717945098876953, -0.775345504283905, -0.37359100580215454, 0.4742753505706787, 0.6020373702049255, -1.9784806966781616, 0.20151785016059875, 0.792227029800415, -0.6853451728820801, -0.9378445148468018, 1.2917885780334473, 0.5118474364280701, -0.6703919768333435, -0.13726922869682312, 0.061843860894441605, 0.1113542914390564, -0.21189947426319122, 0.7923153042793274, -0.26849091053009033, -0.4206104278564453, 0.555181086063385, -0.8993574380874634, 0.7088273167610168, -1.2721285820007324, 0.24623258411884308, -0.2197839319705963, 0.35185518860816956, -0.5202930569648743, 1.1285161972045898, 0.3347519338130951, -1.2265719175338745, 0.5811410546302795, 0.0709981769323349, 0.11259571462869644, -0.6052687168121338, 0.3956666886806488, -1.4177360534667969, -0.47154486179351807, 0.7829407453536987, 0.7812595367431641, 0.409868061542511, -1.0705686807632446, -0.14458966255187988, 0.19218018651008606, -0.9938665628433228, -0.9246885180473328, 0.6882327795028687, 0.7232897877693176, 0.9742856621742249, 0.9421583414077759, -2.013655424118042, 0.5138810873031616, -0.35387474298477173, -0.053638819605112076, -0.49056920409202576, 0.764045774936676, 0.5731780529022217, 1.856479525566101, 1.443498969078064, -0.05375462397933006, 0.5416580438613892, 1.2037427425384521, 0.11307643353939056, 0.7939389944076538, 0.9238234162330627, -2.1277246475219727, -0.47789105772972107, -0.162346750497818, 0.5785956978797913, 0.5801120400428772, -0.058625902980566025, -1.395417332649231, -0.6375816464424133, 0.020835362374782562, 1.058537483215332, -0.018848678097128868, -0.04107968509197235, -3.136934995651245, -0.26386454701423645, -0.1791820526123047, -0.939464271068573, -0.8862547278404236, -0.5234784483909607, -0.6985154747962952, -0.1375887244939804, -0.5294190049171448, -0.37328439950942993, -0.5210985541343689, 0.8781639337539673, 1.293804407119751, 0.21431924402713776, 0.4673939645290375, 2.2318196296691895, 0.5414448380470276, -0.09591147303581238, -1.1569947004318237, 1.4510467052459717, 0.7384700179100037, -0.40101712942123413, 0.5916369557380676, -1.077917456626892, -0.19602371752262115, 0.26409241557121277, 0.0546494722366333, -0.6651401519775391, 0.3686176538467407, 0.2542794346809387, 0.08412809669971466, 0.4313124716281891, 1.71140718460083, -0.7037523984909058, -1.5048351287841797, -0.8993979692459106, 1.9160889387130737, -1.179302453994751, 0.590650737285614, -1.325411319732666, -0.30780261754989624, -1.4524534940719604, -0.8152384757995605, 0.33646661043167114, 1.052293062210083, -0.860176146030426, -1.4035922288894653, -1.6111459732055664, 1.334827184677124, 0.28125908970832825, -0.06371207535266876, 1.1030751466751099, 1.1650482416152954, -0.25822558999061584, -0.3305226266384125, -1.211885690689087, -1.9328312873840332, 0.45592960715293884, 0.6179183721542358, -0.327641099691391, -0.20418532192707062, 0.5646540522575378, -1.4506847858428955, -0.047504786401987076, 1.1617767810821533, 0.5069015622138977, 0.8317010998725891, -0.3635669946670532, 0.043889712542295456, -0.49458953738212585, 0.6325579285621643, 0.3908974230289459, 0.9220861792564392, 0.6067058444023132, -1.2613444328308105, -1.1087373495101929, -0.5426913499832153, 0.0468275249004364, -0.871803343296051, -1.177016258239746, 0.31222835183143616, -1.0536177158355713, 0.44913026690483093, -0.7253422737121582, -0.12385772168636322, -0.0032385485246777534, 0.13311783969402313, 1.0303153991699219, 0.5114327073097229, -2.984279155731201, 0.7182751893997192, -0.49994757771492004, 0.3933054208755493, 0.30541783571243286, -1.0250277519226074, -0.7691435217857361, 0.13800056278705597, -0.46832340955734253, -0.013304117135703564, -1.0538363456726074, -1.2880628108978271, 0.7307272553443909, 1.131606101989746, -2.6820404529571533, -0.4040535092353821, -0.5244058966636658, -0.7304015755653381, 0.13060560822486877, -1.6792553663253784, 1.7894128561019897, -1.6619454622268677, 0.1609540432691574, -0.21125611662864685, 1.546432614326477, 2.4631106853485107, 0.499411016702652, 0.298654168844223, -0.08051931113004684, 1.6371883153915405, -0.7794657349586487, -0.43127405643463135, -0.6146175861358643, -0.6682436466217041, 1.5497394800186157, -1.2172390222549438, 0.15100444853305817, 0.3185919523239136, 0.046056464314460754, 1.551344871520996, 0.10528399795293808, 1.4738407135009766, -0.672396719455719, -0.37902480363845825, 0.9611265063285828, -0.09459897875785828, 1.976765513420105, 0.7500467896461487, -0.043848391622304916, 2.653858184814453, 0.4233679473400116, 0.602514922618866, 1.4624977111816406, -0.5809413194656372, 2.0373096466064453, -0.7150929570198059, 0.7796493768692017, -0.7743375897407532, 1.6968085765838623, -1.2241015434265137, 1.77755606174469, 0.07890079915523529, -0.0014932230114936829, -1.8362107276916504, 0.8646175265312195, -0.7539069056510925, 1.0180397033691406, -0.579189658164978, -0.45297864079475403, 0.6501564383506775, 0.8137108087539673, 0.061858948320150375, 0.7997409701347351, -0.5677639842033386, 1.9040656089782715, -0.23688039183616638, -0.5275804400444031, -0.5499448180198669, 0.31199389696121216, -0.6530529856681824, 0.1383497416973114, -0.3914811909198761, 1.0143591165542603, -0.9919866323471069, -0.05246203765273094, 0.6509339213371277, 0.37123486399650574, 0.09893644601106644, 0.19489175081253052, 1.2371543645858765, -1.291335105895996, -2.459599256515503, 0.051804013550281525, -0.2413257658481598, 2.411716938018799, 1.2190645933151245, -0.14890561997890472, -0.02437758818268776, 0.24363018572330475, 0.6197311878204346, 0.01119688805192709, -0.5944152474403381, 1.877220630645752, 1.8220298290252686, -0.12989814579486847, 0.5786577463150024, -0.5456134080886841, 1.1900449991226196, 1.3876991271972656, -0.06909461319446564, -0.7103807926177979, -0.7908427119255066, 1.624927043914795, -0.4705951511859894, -0.882859468460083, -0.2969982922077179, -1.566750168800354, 0.16099506616592407, 0.12262105941772461, 0.20099376142024994, 1.5937179327011108, 1.0780646800994873, -0.5320669412612915, 0.0487443208694458, -0.7177615165710449, -1.509425401687622, 2.157473087310791, 0.5436121821403503, -0.27720552682876587, 0.02396317385137081, -0.48899927735328674, -1.682563066482544, 1.2260102033615112, -0.21228957176208496, 0.7796583771705627, 1.403779149055481, 1.1711747646331787, -1.1010764837265015, 0.3156706988811493, 1.8171420097351074, 1.5493104457855225, -1.0474879741668701, -1.0303529500961304, -0.3919953405857086, 0.9831132888793945, 0.34624412655830383, -0.4816104471683502, 0.5254644751548767, 0.08450672030448914, 2.6571786403656006, 0.2715594470500946, -2.5945966243743896, 0.5356812477111816, 0.31764963269233704, -1.1440608501434326, 0.21260885894298553, -1.578357458114624, 0.6204691529273987, -0.16765159368515015, -1.528861403465271, -0.3558623492717743, 0.1664620041847229, -1.3283891677856445, 0.3071015477180481, 1.143874168395996, -1.1141510009765625, 1.4150110483169556, 0.28308770060539246, -0.8434162139892578, -0.27552029490470886, -0.25764212012290955, -1.7024521827697754, -1.1580979824066162, -0.13164681196212769, -1.8095271587371826, -0.8496293425559998, 0.60202956199646, -0.5300661325454712, -0.5001576542854309, -2.120224714279175, 1.5256155729293823, 0.44110363721847534, 0.7889083027839661, 0.7338263392448425, 0.36083611845970154, -0.28565600514411926, 2.0765883922576904, -1.1486181020736694, -0.43942025303840637, -1.5276387929916382, 0.41837137937545776, 1.3801205158233643, 0.05531688407063484, -0.3805256485939026, 0.6253865361213684, 0.4535090923309326, -0.31983682513237, -0.1578454226255417, 1.212592363357544, 0.10205354541540146, 0.992824912071228, -0.5935373306274414, 0.3196616470813751, -1.5253667831420898, -1.3598016500473022, 1.608392596244812, -1.1227846145629883, 0.5178754925727844, -0.7284587025642395, 1.2676805257797241, -1.047908067703247, 1.7625778913497925, 0.5115757584571838, -1.3812141418457031, -1.0255882740020752, -0.8489067554473877, -0.7320059537887573, 0.8439725637435913, -0.5103258490562439, -2.371819019317627, -1.4119046926498413, 1.098320722579956, 0.6678159832954407, -0.4391495883464813, 1.2839887142181396, 0.8195237517356873, -0.4648192822933197, 0.02052055485546589, -0.021473877131938934, -0.2922573387622833, -1.4737943410873413, 0.7967253923416138, -2.1877481937408447, -0.3589009940624237, -1.6365225315093994, 0.12989941239356995, 0.20189791917800903, -0.38773369789123535, 0.5682162046432495, -1.5597087144851685, -1.677510380744934, -1.0655332803726196, -2.276871681213379, -1.631325602531433, 0.5777907371520996, -0.495505154132843, -0.3940173387527466, 0.4170646667480469, -1.3968122005462646, -0.9390032291412354, 0.4427558481693268, 0.9364991784095764, 1.0186660289764404, 0.30013060569763184, -0.9469718933105469, 0.07360555231571198, -0.5482607483863831, 0.8437169194221497, -0.9811174273490906, 0.05943036824464798, 0.7174813747406006, 0.9869445562362671, -1.214395523071289, 0.8074359893798828, -0.010107259266078472, 2.773435592651367, 1.0817830562591553, 0.8423680663108826, 0.4322001039981842, -1.1792683601379395, 1.1239657402038574, 0.912803590297699, 0.5936126112937927, -1.779221773147583, -1.9121921062469482, -1.264106273651123, -0.1587551236152649, -1.7912046909332275, -0.06607624888420105, 0.6463915109634399, -0.7041670680046082, 0.5710224509239197, 2.2527003288269043, 0.23677614331245422, 1.1062209606170654, 0.166448175907135, -1.6177335977554321, 0.1311511993408203, 0.909609854221344, 0.774665117263794, -0.015258139930665493, -0.12042149901390076, 0.5364617705345154, 0.6648809909820557, -0.036872342228889465, 1.1651546955108643, -0.12140507996082306, -0.2369438260793686, -0.2052116096019745, 1.0886191129684448, -1.5626415014266968, -1.1489208936691284, -1.1109415292739868, -2.619293212890625, 1.7595698833465576, -0.9730729460716248, -0.26731953024864197, -0.0642426386475563, 0.6323125958442688, -0.634929895401001, 0.3052970767021179, -0.3965371549129486, 1.3789513111114502, -1.1492054462432861, -1.6695843935012817, -0.31403684616088867, -1.043500542640686, -2.78303599357605, 1.0349735021591187, 0.6438396573066711, 0.7969856262207031, 3.0384397506713867, 0.10884193331003189, -1.6346752643585205, 0.9939011931419373, 2.3847317695617676, -0.32803457975387573, -0.43337690830230713, -0.6180703043937683, 0.9738391637802124, -0.9324089884757996, -0.49938511848449707, -0.46356624364852905, 1.519142985343933, -0.905137300491333, -0.9812606573104858, 2.0058014392852783, -0.22032764554023743, 0.018007969483733177, -0.7032260894775391, -0.2887561619281769, 0.544910728931427, 0.2669675350189209, -0.4520249664783478, 1.345349669456482, -0.06868858635425568, 1.1815425157546997, -2.231449842453003, -0.5295534729957581, 0.31643831729888916, -0.44043052196502686, -0.17026522755622864, -0.15804901719093323, 0.46785056591033936, 1.0063378810882568, 0.5830512642860413, 0.1190003827214241, -0.11962692439556122, 0.5255104303359985, -1.2922217845916748, 0.8098034262657166, -0.44181668758392334, 0.01847786083817482, 0.5418106913566589, -0.33440837264060974, -0.2765198349952698, -1.1879292726516724, 1.161941409111023, -1.3559564352035522, 1.02411949634552, 0.09314994513988495, 0.7662911415100098, -1.919799566268921, -1.0650560855865479, 1.1154037714004517, -0.3713015019893646, -1.555641770362854, -0.20183804631233215, 0.5887831449508667, 1.0146688222885132, 0.36623212695121765, 1.880548357963562, -0.13437296450138092, 2.0392463207244873, 0.09954539686441422, -0.8357815146446228, -1.2578625679016113, -1.5893726348876953, 1.4885764122009277, 0.8690657615661621, 0.4309345483779907, 0.4608408510684967, -0.7748932838439941, 1.030169129371643, -0.07062014192342758, 0.3419029712677002, 0.42391112446784973, -0.5522593259811401, -0.25771448016166687, -1.0348104238510132, -0.8694587349891663, -0.07338570058345795, 2.920860767364502, 0.029700839892029762, -0.4360882341861725, 0.1387544572353363, 0.4244551360607147, -0.6374909281730652, 0.4256758391857147, -1.1602648496627808, -0.21788577735424042, 1.1693781614303589, 0.39201727509498596, 0.4945770800113678, 1.8907010555267334, 0.3406122624874115, 0.8922291398048401, -0.3696843981742859, 0.023089023306965828, 0.6369585990905762, -1.5738166570663452, -0.09126242995262146, 0.6699671149253845, 1.1088085174560547, 1.7663886547088623, 0.6743031144142151, -1.2739999294281006, -0.45631080865859985, 1.3787494897842407, 2.243349075317383, 0.34461939334869385, -0.40946856141090393, -1.1705265045166016, -0.4547736346721649, 1.4208910465240479, -0.27495667338371277, 0.7329989671707153, -0.04231517016887665, -0.08603768795728683, 0.009676781482994556, -1.9341179132461548, -0.2890014052391052, -0.5532861948013306, 0.5650802850723267, 2.1430130004882812, -0.06626460701227188, 0.8601999878883362, -1.6554338932037354, -0.5309112071990967, -0.19462037086486816, 0.8270055055618286, -0.5151770710945129, -0.9641947746276855, 0.2725769877433777, -0.640411913394928, -1.2140331268310547, 1.2857803106307983, 0.3588978052139282, 0.9099138975143433, -0.3777756690979004, 1.1016191244125366, 0.10228132456541061, 1.282028317451477, 0.24516859650611877, -0.7766613364219666, 0.7136430144309998, -0.11363302916288376, 0.48922932147979736, 0.6254209280014038, -1.5687731504440308, 0.2079416960477829, 0.09657758474349976, 0.036730099469423294, 0.6170011758804321, 1.1280866861343384, -0.1116035059094429, 0.41659799218177795, -0.8174919486045837, -1.1078729629516602, -0.26680755615234375, 0.7993113398551941, -0.5321096777915955, 1.1873916387557983, -0.9339505434036255, -1.5311682224273682, 1.2095452547073364, -0.3551800549030304, 1.2138795852661133, 1.860430121421814, 0.6669473648071289, -1.9167619943618774, -0.21966113150119781, -0.015319185331463814, 1.2557661533355713, 0.09512386471033096, -1.5876156091690063, 0.5210635662078857, -0.6002453565597534, -0.20322884619235992, -0.9190101027488708, 0.0626000165939331, -1.8056167364120483, 0.9103989005088806, 0.5512737035751343, -1.0384594202041626, -1.511482834815979, 2.37374210357666, 0.39255663752555847, -2.1266465187072754, -1.3725218772888184, -1.7159584760665894, 0.4410873055458069, -0.26374226808547974, 0.10825436562299728, 1.0114368200302124, 0.8877831697463989, -0.053053319454193115, -0.4870978593826294, -0.724269688129425, -0.45537710189819336, -0.3475472927093506, -0.6639754772186279, -0.6043525338172913, 1.4762192964553833, -1.4855296611785889, -0.684522807598114, -1.1096299886703491, -0.8222641348838806, -1.4532735347747803, -0.22788476943969727, 0.47719138860702515, -0.7006999254226685, -0.9768746495246887, 0.9993487000465393, -0.41357752680778503, -1.0525257587432861, -0.5752292275428772, 0.23800601065158844, 0.047451019287109375, -0.8836626410484314, -0.22157453000545502, 0.548102855682373, -0.03195732459425926, -0.14436590671539307, 0.6835616230964661, 0.1731451451778412, -0.5225206017494202, 1.2706775665283203, -0.3527219593524933, -0.23557384312152863, -0.42456531524658203, -0.71805739402771, 0.2720811367034912, -0.4709726870059967, 0.7261255979537964, 1.4794219732284546, 0.17659690976142883, -1.4101471900939941, -0.9952163100242615, -1.3375575542449951, -0.5829481482505798, -0.3903784155845642, 1.6081827878952026, -0.5014714598655701, -0.718755304813385, 1.6480679512023926, 0.16237647831439972, 0.9006698131561279, 0.3211381435394287, -1.5809340476989746, -0.33224645256996155, -0.013708931393921375, -0.8942122459411621, 0.6802228093147278, 0.45111092925071716, -0.6343186497688293, -1.5712779760360718, -1.2456841468811035, 0.929882824420929, -1.1380764245986938, -0.37531378865242004, -1.2177571058273315, 0.625650942325592, 0.551784873008728, 1.1711622476577759, -0.25663521885871887, -0.331996351480484, 1.2544490098953247, 0.0031108222901821136, 0.7000959515571594, -0.43085533380508423, 0.9462567567825317, 0.8312712907791138, 0.15359316766262054, 0.19992096722126007, -1.1697601079940796, 0.17661452293395996, 0.12325794994831085, -0.05841777101159096, -0.8590970039367676, 1.3125314712524414, 0.09618736058473587, 0.6643317341804504, -0.1405920833349228, 0.8029902577400208, -0.976814329624176, 1.393766164779663, 2.0881922245025635, 0.29292774200439453, 0.6050899028778076, 0.7656151056289673, -1.1843196153640747, -1.6704986095428467, 0.148893803358078, 0.5224205255508423, -0.5251521468162537, 2.650857925415039, -0.37710025906562805, -0.7348009943962097, -0.11152465641498566, -0.8741008639335632, -0.5899892449378967, 0.3476794362068176, -0.30989548563957214, -0.1596481055021286, 0.8035088181495667, 0.23949161171913147, 0.24609875679016113, -0.515364944934845, -2.2490756511688232, -0.26128020882606506, -0.15838204324245453, 0.6296007633209229, 0.21581053733825684, -2.2893104553222656, -1.4004825353622437, -0.6649665236473083, 0.7006224393844604, -0.204290509223938, 1.2599265575408936, 0.7925756573677063, -0.26625946164131165, 2.369630813598633, 0.6584379076957703, 0.6210501790046692, -0.5071987509727478, 1.10608971118927, 0.43968063592910767, 1.060312271118164, -0.9854183793067932, -1.49836003780365, 0.20200252532958984, -1.3845425844192505, -1.5975035429000854, 1.5795284509658813, 0.809638500213623, -0.22183169424533844, 1.1826093196868896, 0.6581499576568604, 0.22313830256462097, -0.6632924675941467, 0.9232058525085449, 0.389103502035141, 0.8508009910583496, 0.20180442929267883, -0.9454810619354248, -0.6575464010238647, -1.647784948348999, -0.014111983589828014, 1.9580869674682617, 0.8225141167640686, 0.056979261338710785, -0.5197111964225769, 2.6320204734802246, 0.31725913286209106, 0.804288387298584, -0.12794309854507446, 1.9660841226577759, -1.0885111093521118, -0.3210047483444214, 0.8204889893531799, 0.9320843815803528, -0.6190321445465088, -0.4695040285587311, -0.4206030070781708, -0.07240568846464157, 0.41182056069374084, -1.2403817176818848, -0.7608641982078552, 0.2983650267124176, -0.500580906867981, 0.03468677029013634, 0.8360025882720947, -1.0005381107330322, 2.187680244445801, 0.8818342089653015, -0.36525020003318787, -1.5634368658065796, 1.4039695262908936, 2.1419408321380615, 0.18094992637634277, 0.2867739200592041, -0.5984063148498535, -1.1825836896896362, -1.4130066633224487, 0.4642539918422699, -0.4250178635120392, -0.8843587040901184, 0.06438565999269485, 0.5981051921844482, -0.7937089204788208, -1.37485671043396, 0.23931866884231567, -0.3182176351547241, -0.39732083678245544, 0.7202773690223694, -0.20764322578907013, 0.8881052732467651, -0.142914280295372, 1.1453527212142944, 0.6130934357643127, 1.2131786346435547, 1.3864681720733643, 1.4940658807754517, 1.0610406398773193, 1.6167961359024048, -0.49625101685523987, 0.46771299839019775, 1.1942685842514038, -0.1625780314207077, 1.8773809671401978, 0.11247004568576813, 1.7539546489715576, 0.4475429654121399, -1.359615683555603, 0.8896015882492065, 0.40834471583366394, 0.4752541482448578, 0.5688403844833374, -0.4476885497570038, 0.8532087206840515, 1.9044690132141113, -0.6244862079620361, 0.1627432107925415, -0.15053848922252655, -0.6929864287376404, 0.029884573072195053, -1.0447776317596436, -4.142892837524414, 0.3962870240211487, 0.5113449096679688, -1.272776484489441, 0.5140483379364014, 1.1144790649414062, -0.0590120367705822, 0.9372459053993225, -1.8151601552963257, -0.8996393084526062, -0.7978890538215637, -0.43651139736175537, 0.367051362991333, -0.7463590502738953, -1.4669185876846313, 1.4874809980392456, 0.15006589889526367, 0.9151869416236877, -1.423195481300354, -0.2168419063091278, -0.29688137769699097, 0.34826043248176575, -1.2375084161758423, 0.42329397797584534, -0.7328113913536072, 0.7644944787025452, -1.0517470836639404, -1.111405611038208, -1.349912405014038, -0.1296597272157669, -0.2997928261756897, -0.04204820841550827, 0.3904150426387787, 0.13475407660007477, 1.3324997425079346, 1.0237395763397217, -0.06971493363380432, -1.0241801738739014, -0.45591244101524353, 2.2831599712371826, -1.2047057151794434, 1.2058194875717163, 0.4596542418003082, 1.854884147644043, 0.6898966431617737, 0.8859049677848816, 0.42295846343040466, 0.23297153413295746, 0.3070991337299347, -0.9394948482513428, -0.40527641773223877, -1.670099139213562, -0.011066116392612457, -0.11506311595439911, 0.7483073472976685, -0.7059043645858765, 1.202562689781189, -0.2855715751647949, 0.3832045793533325, -1.5439659357070923, -3.3556408882141113, -0.22836744785308838, -0.4413950443267822, -1.0684963464736938, 0.2393260896205902, 1.003353238105774, -0.6923230290412903, -0.8754499554634094, -2.6639187335968018, 1.178684949874878, -0.9883714914321899, 0.5431759357452393, 1.0256712436676025, 0.14708282053470612, 0.8615066409111023, -0.5075947642326355, 2.3840792179107666, -1.5633020401000977, 0.6086937189102173, 1.5859557390213013, -0.21740449965000153, -1.171404480934143, 1.9427844285964966, 0.7937604188919067, 0.027392111718654633, 1.5102968215942383, -0.37008845806121826, 0.4977470636367798, 1.8652223348617554, 1.6203608512878418, 1.1994138956069946, -2.436523199081421, -2.157909870147705, -0.09507399797439575, -0.2684977054595947, -1.2211265563964844, -0.11562734842300415, -0.08119917660951614, 0.7737898826599121, -0.21918946504592896, 0.5010015964508057, -1.1355546712875366, -0.6232270002365112, -0.47064462304115295, -0.8957529067993164, -1.136396050453186, -1.0930638313293457, 0.7667883634567261, 2.017474412918091, 0.6555522680282593, -1.1024839878082275, -2.70027756690979, 1.0486879348754883, -2.2607200145721436, 0.6017343401908875, 1.433159589767456, 0.3091994524002075, 0.19443906843662262, -0.0010172298643738031, -0.7914078831672668, -0.4354965388774872, 1.0922136306762695, 0.2592522203922272, -1.423709511756897, -1.2280446290969849, -0.48452028632164, 0.7808070182800293, 1.1163767576217651, 0.3080785274505615, -1.1199623346328735, -0.9014104008674622, 1.5944348573684692, 0.809939980506897, -0.48293375968933105, -0.33627891540527344, -0.728507936000824, -1.6126054525375366, 1.1225595474243164, -0.3788875639438629, 1.0835636854171753, -0.3870052397251129, 0.4277888238430023, -0.32474201917648315, 0.38045287132263184, 0.33643344044685364, 1.8236804008483887, 0.02195185422897339, 0.6396594047546387, -0.8315185904502869, 0.9028589129447937, 0.2646484375, 0.4133739769458771, 2.5179953575134277, -0.6095598340034485, 1.0571420192718506, -0.2597944736480713, 0.7914844751358032, 0.6742269396781921, -1.0298614501953125, -0.2571885585784912, -0.7445533871650696, -0.441200852394104, 0.956254780292511, -0.6483919024467468, -1.194089651107788, -1.2226039171218872, 1.3435635566711426, 1.89577054977417, -1.7449721097946167, 0.5725422501564026, -1.185044765472412, 0.6756964325904846, 1.872161865234375, 0.12763619422912598, 0.09371678531169891, 0.0002478938549757004, -0.7400015592575073, 0.7697232961654663, 0.3694694936275482, -0.08440106362104416, 0.45002299547195435, 0.0029966768343001604, -0.202165424823761, 1.771677017211914, 1.4429872035980225, -0.8079982399940491, 0.9819099307060242, 1.5098068714141846, -1.5301371812820435, -0.7759853601455688, 1.0614243745803833, -0.7944525480270386, 1.4295542240142822, -1.134936809539795, -0.6331723928451538, -1.1843936443328857, 0.1556926965713501, 0.5181301236152649, -0.13001573085784912, -0.1274692565202713, -0.13095197081565857, 1.2135053873062134, -0.46154820919036865, 1.6935946941375732, -1.2934832572937012, -2.257289171218872, 0.7352107167243958, 0.3363621234893799, 0.5035889148712158, -1.9600789546966553, -0.116851806640625, 0.24098125100135803, 0.6374961733818054, -0.5010445713996887, -0.04744747281074524, -1.6140594482421875, 0.6441494226455688, -1.6859663724899292, -0.4554097354412079, 0.3890719413757324, 0.25478458404541016, -1.5101425647735596, 1.1214394569396973, -0.6312149167060852, 0.6077934503555298, 0.09128667414188385, 1.1393686532974243, 0.19624188542366028, 1.0568516254425049, 0.46102669835090637, -2.0165534019470215, -0.40768373012542725, -0.5731223821640015, -1.642088770866394, -0.8360973596572876, 0.014604187570512295, -1.0958796739578247, -0.6016586422920227, 0.4863761067390442, 0.9888652563095093, 0.7109102010726929, -1.5103248357772827, -0.44806408882141113, 0.17139562964439392, 1.5170420408248901, -0.4820137321949005, 0.538281261920929, -0.07965502887964249, 0.7800680994987488, 1.7613285779953003, 1.8490607738494873, 1.2002801895141602, 0.7924781441688538, 0.1728549301624298, -0.2870509922504425, -1.011496663093567, 0.7127639651298523, -1.347782850265503, -0.44908493757247925, 0.46984952688217163, 0.2671419680118561, 0.9365683794021606, 0.16684465110301971, 0.45046356320381165, -0.369055837392807, 0.6756237149238586, -0.8155454993247986, -0.6643508672714233, -0.40437766909599304, 0.06590622663497925, 0.6496062874794006, -1.666608214378357, -0.03320574387907982, 0.9185694456100464, -1.1970049142837524, -1.4099416732788086, 0.7543562650680542, 0.48827993869781494, -0.016100449487566948, 0.347226619720459, -0.16062422096729279, 0.44398507475852966, 0.7907341718673706, -0.690947949886322, 0.37956178188323975, 1.5506330728530884, -0.00451289489865303, 0.6141844391822815, -0.7418960928916931, 0.49876293540000916, 1.9998390674591064, -0.8088881373405457, -1.200905442237854, 0.14089372754096985, -2.0072999000549316, 1.0752830505371094, -0.9413755536079407, 0.144863098859787, 0.035094160586595535, -0.6293017864227295, 0.36748114228248596, -2.570842981338501, 0.26795148849487305, 0.3312157094478607, 0.8274869918823242, -0.6744167804718018, 1.3869892358779907, 1.5380569696426392, 2.0845375061035156, 0.7086749076843262, -0.8730274438858032, 0.7581827640533447, 0.4481918513774872, -0.2039545178413391, -0.8457404375076294, -0.019914187490940094, 0.6310129761695862, -0.15400046110153198, -0.4597492218017578, -0.30060479044914246, 0.7919981479644775, -0.9863638281822205, -0.034884482622146606, 1.1483557224273682, -0.8903233408927917, -0.3343416452407837, 0.1857125163078308, 0.507797122001648, -0.5443049073219299, -2.8973147869110107, 0.9881216883659363, 1.310128092765808, -1.9037514925003052, -0.01620291732251644, 0.11143700033426285, -0.8976759314537048, -0.8106405138969421, 1.3020424842834473, -0.7425699830055237, 1.2048885822296143, -0.03223133087158203, 0.046475738286972046, -0.714488685131073, -0.04586213454604149, -0.21306473016738892, 0.7057672142982483, 0.29028937220573425, 2.33280873298645, 0.04960104823112488, -0.3548282980918884, 0.1403520256280899, 1.1027990579605103, 0.8443953394889832, 1.564976692199707, -0.07894589751958847, -0.5592381358146667, 0.8067716360092163, -1.1183762550354004, 0.41460171341896057, 0.629014790058136, 0.3239023983478546, -2.520129442214966, -0.6046906113624573, 0.7380004525184631, 0.5423578023910522, 0.9791197776794434, 1.3756409883499146, -0.29204729199409485, -0.6410121917724609, -2.089688301086426, -2.2529563903808594, -0.19471120834350586, -0.7923608422279358, -0.7781155109405518, 1.2923496961593628, -0.03560435026884079, -0.3670773208141327, -0.0006406275206245482, 1.0093951225280762, 0.3293763995170593, -0.7519626021385193, 0.2563881278038025, -0.896753191947937, 0.5634214282035828, -0.6039738059043884, 0.10715968906879425, -0.36460137367248535, -0.3594221770763397, 1.2658987045288086, -0.4277297556400299, -0.18553557991981506, 1.0384770631790161, -1.1271867752075195, -2.1274032592773438, 2.1335933208465576, -1.808226466178894, 0.15296919643878937, 0.30971449613571167, -0.25296327471733093, 0.6522631049156189, -0.06256470084190369, -0.7283874154090881, 0.8073043823242188, 0.8241995573043823, 0.9556503295898438, 0.01938835345208645, -0.01025813352316618, 0.9343569278717041, -0.927843451499939, 0.10815012454986572, -0.24339918792247772, -0.3353661298751831, -0.546403706073761, 1.1351537704467773, 0.2789127826690674, 1.286895751953125, 0.9171270728111267, 0.6945757269859314, -0.1366063356399536, -0.3678368330001831, 0.8561084270477295, -1.112494945526123, -0.05626698210835457, -0.9511529803276062, 0.3113681972026825, -2.4575533866882324, 0.9991410374641418, -0.10910243541002274, 1.428323745727539, 0.21235471963882446, -1.5098940134048462, -0.27132219076156616, -0.05986863374710083, -0.38649725914001465, -0.3194062113761902, 1.3045350313186646, -0.3559914231300354, 1.4290581941604614, 0.6766263246536255, 0.9169262647628784, 0.26089978218078613, 0.24348151683807373, 0.2623614966869354, -0.2508600354194641, 1.232595682144165, -0.8868474364280701, -0.3803896903991699, 0.7746403813362122, -0.2279006689786911, -1.0462850332260132, 0.13909325003623962, 0.8810215592384338, -0.05653052404522896, 1.2593892812728882, 0.38467732071876526, 1.4312236309051514, -0.481614887714386, 0.1969088315963745, 0.5462742447853088, 1.6046464443206787, 0.14609964191913605, -0.3858894109725952, -0.10530741512775421, -1.4619685411453247, 1.414811611175537, -1.2891550064086914, 1.1716973781585693, -0.4490851163864136, 0.5076953768730164, -1.1751437187194824, 0.7985503077507019, -0.2149077206850052, -1.176506757736206, 0.9981604814529419, 1.7643159627914429, -2.127403974533081, -0.9228654503822327, -0.21613627672195435, 1.63971745967865, -0.018215594813227654, 0.6140064001083374, 0.13754169642925262, 0.6976073384284973, 0.8016719818115234, 0.0012730908347293735, -0.26339244842529297, 0.40268176794052124, 2.1402297019958496, -0.531008243560791, -0.880462646484375, 0.681259036064148, 1.8206286430358887, -1.2048672437667847, -0.06930690258741379, 0.23156630992889404, -1.404494047164917, 0.0015493083046749234, -1.2005300521850586, 0.6984289884567261, -2.760331392288208, -0.3611833155155182, -0.4518863558769226, -0.29916366934776306, -0.2266051024198532, 1.5230779647827148, -1.31452476978302, 0.19989150762557983, -0.3287491798400879, -0.6192755103111267, 0.025731053203344345, -0.08546623587608337, -0.7907305955886841, -0.5077592134475708, -0.34274762868881226, 2.0441553592681885, 0.35804301500320435, -1.2620960474014282, 1.9692169427871704, 0.6224166750907898, 0.4694894552230835, -1.5349785089492798, -1.0522774457931519, -0.1231301948428154, 0.5539934039115906, 0.9732133746147156, 0.02095138467848301, 0.13764561712741852, -0.16570483148097992, 1.6666642427444458, 2.3133037090301514, -0.5132427215576172, 1.4352432489395142, -1.0978267192840576, 2.0333194732666016, 1.3298187255859375, -1.0023189783096313, 0.8491287231445312, 1.0987745523452759, -0.2792922258377075, 0.34227922558784485, -1.823023796081543, 0.8876863121986389, -0.7149918079376221, 0.6798567175865173, -0.29943859577178955, 0.8126136660575867, -1.168445110321045, -0.9238936901092529, 0.01620417647063732, 0.07828274369239807, -0.5781503319740295, 1.5075737237930298, 0.45134633779525757, -0.0931711345911026, -0.22919707000255585, -0.04523346200585365, 0.9730731248855591, 0.3653322458267212, -0.4075479805469513, 0.8485914468765259, 0.2765117287635803, 0.6600531339645386, 1.3539317846298218, -0.41563892364501953, -0.48002347350120544, 0.415656179189682, -1.1240073442459106, -1.6356834173202515, 2.6412875652313232, 1.2081090211868286, 0.15697751939296722, 1.2893495559692383, 2.347929000854492, -1.7572671175003052, -0.5905064344406128, 0.9949122071266174, -0.805433988571167, -1.6959508657455444, 1.884080171585083, -0.19466634094715118, -2.2177841663360596, 0.48019570112228394, 0.6322563886642456, 0.3959534764289856, 1.674058198928833, 0.788800060749054, 0.5096863508224487, 2.2044105529785156, 0.10403569042682648, -1.864261269569397, 2.1386325359344482, 0.6619711518287659, 1.3949140310287476, 0.06614010781049728, -0.1747215986251831, 0.5798646807670593, -0.19299432635307312, -1.4700140953063965, -0.42112451791763306, 1.7654509544372559, -2.024005651473999, 1.1220629215240479, -1.5177383422851562, -0.8271484375, 1.6660070419311523, 0.7889175415039062, -0.9936334490776062, 0.14766590297222137, 0.4793589413166046, 0.2807143032550812, 1.644921898841858, -0.5333935022354126, 1.3468080759048462, -0.8643119931221008, -0.11152118444442749, 0.9252550601959229, -0.7863661050796509, -0.3179624378681183, 1.8364508152008057, 1.1625045537948608, 0.02581683173775673, -0.6319531798362732, -2.217270612716675, 0.4489882290363312, 1.9072835445404053, 1.3483519554138184, 0.19368384778499603, -2.251121759414673, -0.9546146392822266, 0.12665341794490814, 1.7565479278564453, 0.9281153082847595, -0.4316577613353729, 0.19119974970817566, 0.06424355506896973, -0.7900329828262329, 1.487154245376587, 0.1373928189277649, -0.12114991992712021, 1.3744661808013916, 0.6807348728179932, 2.0343005657196045, 1.959626317024231, -0.31565964221954346, -0.790337324142456, 0.3545316159725189, 0.8379015922546387, -0.2143065184354782, 0.9165399074554443, 1.0212053060531616, -0.9530484676361084, 0.439264714717865, -1.4862216711044312, 0.7623528838157654, 0.27267909049987793, -0.6009895205497742, -2.2584617137908936, -1.1413094997406006, -0.9596871733665466, 1.3260539770126343, -0.43329498171806335, -0.08148687332868576, -0.9130480885505676, 0.33328017592430115, 0.45619770884513855, -0.23489779233932495, -0.026732318103313446, -0.5875785946846008, -0.4580317735671997, 0.1328461617231369, -1.5048717260360718, -0.7723105549812317, 1.2190924882888794, 0.7266045808792114, 1.3534518480300903, 1.3199713230133057, -1.2238701581954956, -1.2376842498779297, 1.3160531520843506, 0.5297082662582397, -1.2208143472671509, 1.035752534866333, -0.20810091495513916, 0.7355637550354004, -0.7345859408378601, -1.2268341779708862, -0.31831905245780945, -0.10650436580181122, -0.019926171749830246, -0.03457003831863403, 0.11762645840644836, -0.6431708335876465, -0.7865480780601501, -0.820065438747406, 0.4874616861343384, 0.4350813031196594, -1.747275948524475, -0.3328559696674347, -1.0785956382751465, -0.5640414357185364, -0.8105863928794861, 1.6748687028884888, -0.6880782246589661, -0.07416024059057236, -0.09940753132104874, 1.6651564836502075, 1.328873634338379, -0.9224055409431458, 0.1320137083530426, -0.2449982464313507, -0.9374421834945679, 0.033779263496398926, -0.02441341057419777, 0.4841831624507904, -1.456792950630188, -0.8691458702087402, 0.13223280012607574, 1.244476318359375, 1.8653427362442017, 0.6381714344024658, -0.8151851296424866, -0.19536766409873962, 0.6026037335395813, 0.29788246750831604, 0.4036315977573395, 1.7401131391525269, -0.10209761559963226, -0.3165624141693115, 1.0262424945831299, 0.07184765487909317, 0.8601362109184265, 1.2784135341644287, 0.5957431197166443, -0.334223210811615, -0.16737250983715057, -2.111635684967041, -0.17559055984020233, -0.860514223575592, 0.9273701906204224, -2.745177984237671, 0.7810372710227966, 0.15735790133476257, 1.6253787279129028, -0.336532324552536, 0.4320394694805145, 2.727720260620117, 0.9540805220603943, -1.360843539237976, -0.2554152011871338, 1.1818419694900513, -0.8857060670852661, 0.11170421540737152, -0.5014263391494751, 0.42849424481391907, -1.5979715585708618, 1.6781102418899536, -0.4626395106315613, 0.8559176921844482, -0.6282505393028259, -0.24098443984985352, -0.14819644391536713, -0.14309990406036377, -0.0019085804233327508, 0.7855135202407837, -0.5707711577415466, 0.20552054047584534, -0.8394688367843628, -0.06268665939569473, 0.16093012690544128, 0.5939096808433533, -0.7649626135826111, -0.28292784094810486, -0.690862238407135, -0.07496645301580429, 0.12125030159950256, -2.4222195148468018, 0.3678942322731018, -0.9851237535476685, -0.4529414176940918, 0.9681932330131531, 0.37313202023506165, -0.5387816429138184, 0.17517119646072388, 1.504833698272705, -0.6921127438545227, 1.1123425960540771, -1.3365192413330078, -1.6830041408538818, -0.27268943190574646, -0.26847198605537415, 0.9198680520057678, 1.2029693126678467, -0.8994889855384827, -0.0242083091288805, 1.37416672706604, 1.8964959383010864, 1.107679843902588, -0.3152746558189392, -1.2663242816925049, -0.2769679129123688, -1.333549976348877, -1.3310319185256958, 0.9161091446876526, 1.289933443069458, 0.14100806415081024, 2.427973985671997, 0.7105132937431335, -2.7233660221099854, -1.876827597618103, 0.02817455306649208, 0.651909351348877, -0.8652737736701965, 0.6780588030815125, 0.12262561917304993, -0.5319697856903076, 0.011565458960831165, 0.7703348398208618, 1.2373307943344116, -0.8505446910858154, -0.7766851782798767, 0.510660707950592, -1.1854515075683594, -0.7026759386062622, 0.8260244131088257, -0.10564937442541122, 0.24667435884475708, 0.7611283659934998, 1.940139651298523, -0.4774130582809448, -0.7090739607810974, -0.6809602975845337, -0.04091505706310272, 0.3563010096549988, -0.46075621247291565, 0.2030511349439621, -1.0588229894638062, 1.139581322669983, -1.0371899604797363, -1.9946142435073853, -0.6986521482467651, -1.040618896484375, -1.3525409698486328, -1.3143564462661743, 1.020343542098999, 0.579323410987854, 1.9229589700698853, -1.0606006383895874, -0.3564987778663635, -0.9299565553665161, 0.5908818244934082, 0.6566822528839111, -2.125248670578003, -0.43129053711891174, 0.24261096119880676, 1.814907193183899, 0.4377182722091675, 0.18400350213050842, 0.21712835133075714, 0.6022863984107971, -0.11644814908504486, -0.3925997018814087, -0.15979403257369995, -1.127928376197815, -0.474760502576828, -1.1651431322097778, -2.4630839824676514, 0.6429571509361267, 0.8600722551345825, -0.7464962005615234, -0.8526380062103271, -0.3422777056694031, -0.19956640899181366, -1.202107310295105, -1.773366928100586, -0.22544783353805542, 0.5255645513534546, 0.021933341398835182, -0.7455909252166748, -0.7147470712661743, 1.0717413425445557, -1.7989006042480469, 0.9507761597633362, 1.458907961845398, 0.8040444850921631, 0.5234661102294922, -0.9466000199317932, 1.266654133796692, 1.5476651191711426, -0.49540942907333374, -2.565000295639038, -0.6841189861297607, 0.6157443523406982, -0.4523784816265106, -1.0892300605773926, 1.2434967756271362, 0.18165360391139984, -0.3051178455352783, -0.8285121321678162, 0.508493185043335, 2.4990479946136475, 0.20179101824760437, -0.14413733780384064, 0.004289539996534586, -0.6355151534080505, 0.9136853218078613, -0.9724056124687195, 2.352790594100952, -0.14022715389728546, 0.17219606041908264, 1.4747169017791748, 0.879091739654541, 1.853067398071289, -0.2993922829627991, 1.8815834522247314, 0.43897706270217896, 1.6600569486618042, 0.3939893841743469, 0.07050718367099762, 0.26806068420410156, 0.8315058350563049, 0.39826199412345886, -1.4062832593917847, 0.4486548602581024, -0.10200434178113937, 0.02284335531294346, -1.0534470081329346, 0.9461220502853394, 1.0307049751281738, 1.1704230308532715, 1.0726014375686646, -1.1194276809692383, 2.055957078933716, -0.6858473420143127, -0.8177565932273865, -0.9627580642700195, -0.22257012128829956, 0.6124448180198669, -0.9622762203216553, 0.9395554065704346, -0.5350800156593323, 2.3128466606140137, 0.9858480095863342, 1.5883175134658813, -0.13097871840000153, -0.5765760540962219, -0.3162594139575958, -1.498800277709961, 1.9249929189682007, -1.6024959087371826, 0.7728211879730225, -1.1039483547210693, 0.8881661891937256, -0.3001960813999176, 1.3932899236679077, 0.6778733134269714, 0.4124707877635956, -0.25426459312438965, -0.3888164758682251, 0.49233198165893555, -0.436357706785202, 0.042649704962968826, -0.5738658308982849, -0.5381892919540405, 1.3408654928207397, -0.8372768759727478, 0.40794482827186584, -1.3699275255203247, -0.3766542077064514, 1.393330454826355, 0.2712465822696686, -0.6036279201507568, -0.26475977897644043, -0.32880380749702454, -0.5991823673248291, -1.602770447731018, -0.023755282163619995, -0.41012975573539734, 1.4086354970932007, 1.3607938289642334, -1.7334688901901245, -0.03057735785841942, -0.7050052881240845, 0.9132573008537292, -2.143373966217041, -1.0391865968704224, -0.8446775078773499, -1.1109137535095215, 1.5371739864349365, 0.8463820219039917, -1.5152523517608643, -1.8051315546035767, 0.14945432543754578, 0.7161437273025513, -0.6432929635047913, 0.9510250091552734, 1.1828949451446533, 0.8829526901245117, 0.015994613990187645, 0.5101067423820496, 1.2990918159484863, 1.6744657754898071, 1.3483619689941406, -0.5826199650764465, -0.4491908550262451, 2.6689441204071045, 1.2097196578979492, 0.5982443690299988, -0.7888362407684326, -0.7060816884040833, 0.6360124349594116, -1.6436823606491089, 0.4641686975955963, -0.09934982657432556, -1.4057334661483765, -0.1711871325969696, -0.7629523873329163, 0.5447525382041931, -0.42822784185409546, -1.3611667156219482, -0.8609482049942017, -0.0482262521982193, 0.041144322603940964, -0.9263004064559937, 0.1879560500383377, 0.8829107284545898, -0.035336315631866455, -1.4630684852600098, 0.2570139765739441, -0.5995134711265564, -0.5494652390480042, -0.43657994270324707, 0.5247608423233032, 2.1299335956573486, -0.8936315178871155, -0.11978750675916672, -0.757912278175354, 0.2623540759086609, 0.07299292832612991, -0.4673457443714142, 0.08173583447933197, 0.3909292221069336, 0.5655691623687744, 1.1393284797668457, 1.8489969968795776, 0.47743332386016846, 0.8147130012512207, -0.6556200981140137, 0.3651801347732544, -0.4464399218559265, 0.7078163027763367, 0.9685305953025818, 0.3161143958568573, 0.6380839943885803, 1.1556768417358398, -1.2849701642990112, -0.6235962510108948, 0.09638942033052444, 0.47479870915412903, 1.5719380378723145, -1.0450040102005005, 1.0762277841567993, -0.3176048696041107, -0.019103240221738815, -0.37884005904197693, -0.7250839471817017, -0.675354540348053, -1.8918962478637695, -0.20170308649539948, -1.0173789262771606, 0.37959185242652893, -0.32916954159736633, -0.5118729472160339, -1.9087165594100952, -1.1872597932815552, -0.4808640480041504, -1.699757695198059, 0.7629604339599609, 1.4602112770080566, 1.1901377439498901, 0.06633606553077698, 1.1445389986038208, 1.453190803527832, 1.4242987632751465, -0.39335575699806213, -0.7357659339904785, 0.12248750776052475, 0.018130796030163765, -0.8193605542182922, -0.6493124961853027, 0.20253536105155945, 0.8230568766593933, 0.5376400351524353, -0.4764457643032074, -0.24663145840168, -1.0916885137557983, -0.7785825729370117, -0.056137070059776306, -0.09711385518312454, -1.7711118459701538, -1.6896644830703735, -0.0517333522439003, -2.058603048324585, -0.4061035215854645, 0.7576276659965515, 0.059616491198539734, 0.597154438495636, 1.0908374786376953, 1.5615270137786865, -0.031139928847551346, 1.4653455018997192, -2.2806901931762695, 0.4211271405220032, -0.7863336801528931, 0.4850245416164398, -1.4937299489974976, 0.7913224697113037, 0.5646535754203796, 1.0994389057159424, 0.0869632139801979, 0.5073509216308594, 0.20712167024612427, -0.5660483241081238, -1.7293509244918823, -0.5570120215415955, -0.06135933846235275, -0.2721734046936035, -0.3016410171985626, 0.17280466854572296, -1.0002286434173584, -0.804252028465271, 0.9992510080337524, 1.0774626731872559, 0.35835936665534973, -1.4015518426895142, -1.317175030708313, 0.051798418164253235, 0.8200840950012207, -0.010750740766525269, -1.0323402881622314, 0.041675250977277756, -0.9890487790107727, -1.438886284828186, 0.3488776385784149, 0.10617092251777649, 1.5537906885147095, 1.6617403030395508, -2.1005098819732666, 2.0252010822296143, 0.9772980809211731, 0.14915615320205688, -0.4744172692298889, -0.2460101991891861, 0.641648530960083, -0.7126355171203613, 0.8953030109405518, -2.699345350265503, 0.6657524108886719, -1.8770195245742798, 0.10779442638158798, -0.7702279090881348, 0.18164850771427155, -0.40044718980789185, -0.18378324806690216, -0.6996259093284607, -0.16947320103645325, 0.35472914576530457, -0.09759660810232162, 2.658132553100586, -0.26518017053604126, 0.4279829263687134, -0.7789343595504761, -0.666690468788147, -0.4228556454181671, -0.16349729895591736, 0.31317681074142456, 0.6240442395210266, 0.05868900567293167, 0.22268275916576385, -0.7377824187278748, -0.6317254304885864, 0.2805264890193939, -0.6014538407325745, 0.004634758457541466, -0.7544033527374268, -1.5404927730560303, -0.16025319695472717, 0.4369127154350281, -0.49457406997680664, -0.8528990745544434, 0.14469574391841888, -0.32982271909713745, 0.4671741724014282, -0.14048369228839874, 0.6608936190605164, 0.07181964069604874, -0.4733220636844635, -0.15139611065387726, 0.06001980975270271, -0.11531664431095123, -0.5763081312179565, 0.8931458592414856, -1.360148549079895, -0.496967613697052, 0.761612594127655, -0.9128974676132202, -0.11600285023450851, 0.22749537229537964, -0.8711902499198914, 1.0124294757843018, -0.2525466978549957, -1.159929633140564, 1.1937729120254517, -0.9157571196556091, 0.9964850544929504, -0.6768549680709839, 0.6409260630607605, -1.8568363189697266, 0.18371203541755676, 0.32210734486579895, -0.5494154691696167, -0.32325518131256104, 0.4086429178714752, 0.1545039415359497, -0.402921587228775, 1.2806332111358643, -1.157657265663147, -1.899675726890564, 1.0207939147949219, -0.8444915413856506, 0.26640745997428894, -1.1551390886306763, 0.1515006572008133, 0.17674711346626282, 0.5196755528450012, 1.5725007057189941, 1.2103917598724365, 1.123276710510254, -0.481302946805954, -1.2550129890441895, -0.2693396806716919, -1.1182218790054321, -2.178138017654419, -2.364339590072632, -1.1462160348892212, -2.547215461730957, -1.862416386604309, -0.9219710826873779, -1.4538195133209229, -0.8495011925697327, 0.6993165612220764, 0.724469006061554, -1.6411100625991821, -0.8183522820472717, -0.18066436052322388, 0.5214349627494812, -1.998309850692749, -1.1465915441513062, 0.16687147319316864, -1.6590803861618042, 1.4135174751281738, 1.4798039197921753, -0.23919928073883057, -0.7491230368614197, 1.0678131580352783, -0.23424039781093597, -0.20147699117660522, -0.6166084408760071, 1.7473520040512085, -0.6945587992668152, 1.0091559886932373, -1.0907827615737915, -0.4581615626811981, -0.028662990778684616, 1.4977388381958008, -1.1855624914169312, 1.2894517183303833, -1.6438103914260864, 0.6393311023712158, 0.32853078842163086, 0.010308855213224888, -0.16394169628620148, 0.20613260567188263, -1.743029236793518, 1.5467581748962402, 0.4929562211036682, -0.5902935862541199, 0.9926871061325073, 0.333545982837677, 0.9358084201812744, -0.9324173927307129, -0.6056603789329529, -1.4720277786254883, -0.24240638315677643, 0.1541917324066162, -0.8611970543861389, -0.8013381361961365, 0.7494477033615112, 0.9895848035812378, -0.029187554493546486, 0.7897363305091858, -0.48685821890830994, -0.7626246213912964, 0.3826780617237091, 0.18196024000644684, -0.3635304570198059, -1.273681879043579, -0.6908271908760071, -0.21223817765712738, -0.08967436850070953, 2.107917547225952, 0.17157834768295288, 1.1099083423614502, 0.9382525086402893, 0.5936778783798218, -0.008475970476865768, -0.5374917387962341, -0.3522888422012329, 0.7495669722557068, 1.380946159362793, -0.6852607727050781, -1.0071407556533813, -1.5634942054748535, -1.7858664989471436, -0.48413440585136414, -0.23896822333335876, 0.9651443958282471, 0.6245445609092712, 1.255622148513794, 0.9318568110466003, -0.1095941811800003, 0.15017901360988617, -0.8283446431159973, -0.0850723460316658, 0.41670823097229004, -0.1318991333246231, -0.9202396273612976, -0.24472884833812714, -0.3063102066516876, 1.4588432312011719, -0.11180132627487183, 0.8050262928009033, -0.673841118812561, 0.7173694968223572, -0.9056463837623596, 1.511490821838379, 0.6723135709762573, -1.6969295740127563, -1.280647873878479, -0.17636677622795105, -0.9372451901435852, -0.72261643409729, -0.4726859927177429, 0.48964473605155945, -0.37156423926353455, -0.30468350648880005, 0.5361506342887878, -2.899463176727295, -0.8831079602241516, -0.1014377623796463, 0.1269240379333496, 1.7172491550445557, 2.776428699493408, 2.138648748397827, 1.1143865585327148, 1.3692336082458496, 0.14207395911216736, 1.2510137557983398, 1.4183310270309448, 0.6478189826011658, -0.28813430666923523, 0.23491086065769196, 0.45514771342277527, -0.5762049555778503, 0.548317551612854, -0.18584437668323517, 0.6740930676460266, 0.6290736794471741, -0.505539059638977, -0.17803703248500824, 0.5392629504203796, 0.3403448462486267, 2.062847375869751, 0.05554352328181267, -0.5437158346176147, 1.0659804344177246, -0.8053179383277893, -1.1323113441467285, 0.20097652077674866, 0.8875988125801086, 0.026765819638967514, 1.1608481407165527, -0.08250108361244202, -0.059210218489170074, 1.525559902191162, -0.9675115346908569, 0.037098970264196396, -0.3512212038040161, -0.033558931201696396, 1.331635594367981, 0.02075405977666378, -1.5825275182724, -0.2941559851169586, -0.3175339996814728, -0.9007164239883423, -0.33277133107185364, -1.1873903274536133, 0.5298969149589539, 0.7587436437606812, -2.316981077194214, -1.3925130367279053, 0.11657390743494034, -0.48981693387031555, 0.886313259601593, 0.053363069891929626, 0.31396204233169556, 0.033042632043361664, -0.05921009182929993, -1.0195376873016357, 0.6868868470191956, -0.9026303291320801, -0.2386057823896408, -1.10883629322052, 1.3392943143844604, 0.4265459179878235, 1.4911528825759888, 0.9217057228088379, 1.3644462823867798, -0.14521858096122742, 0.02209506928920746, -1.733309268951416, 0.2847631871700287, 0.006713400594890118, -0.11724899709224701, -0.458854079246521, -1.4911291599273682, 0.17104452848434448, -1.7427493333816528, 0.16439023613929749, -1.9903862476348877, 0.506004810333252, 1.7336711883544922, -1.334565281867981, 0.9369723796844482, -0.41628190875053406, 0.7751415967941284, -0.467563271522522, -1.9137564897537231, -2.313429832458496, -1.0975476503372192, 0.6430553793907166, -0.3142722249031067, 0.45067456364631653, -0.8107560873031616, -1.1414858102798462, -0.6521098017692566, -0.8713625073432922, 0.1613309681415558, -0.5173792243003845, 0.6436232924461365, -0.47275590896606445, -1.0465912818908691, 0.25302475690841675, 1.3341680765151978, 0.6127244830131531, -1.1846446990966797, 0.22019022703170776, -0.635376513004303, -1.3100682497024536, -0.9005608558654785, -0.1367700695991516, -1.0368268489837646, -1.1541630029678345, 0.43653425574302673, -2.7542097568511963, -0.6421101093292236, -0.3805924355983734, 0.0558447390794754, -0.986490786075592, -0.8821921348571777, -1.0562807321548462, 0.9877073168754578, -1.658650517463684, -0.8764442205429077, -2.2051262855529785, -1.0612514019012451, -1.414677619934082, -0.9023053646087646, 0.9196974635124207, 1.3570736646652222, 0.704281210899353, -0.014622303657233715, 0.8564847707748413, -0.31396937370300293, 3.213670015335083, -2.6900885105133057, -0.9600031971931458, -0.8678554892539978, -2.3181326389312744, -0.25750720500946045, 1.6943241357803345, -1.0612268447875977, 1.3152259588241577, -0.37238365411758423, -0.3006848096847534, 0.025629134848713875, -0.07850973308086395, 0.691644549369812, 0.434858113527298, -0.15025992691516876, 0.8771931529045105, -1.9983786344528198, -0.9369648098945618, -0.6788273453712463, 0.7344454526901245, 0.7605552077293396, 0.025318458676338196, -0.0011435338528826833, -0.7264981269836426, -1.874955415725708, -0.14587408304214478, -1.0825527906417847, 1.6259129047393799, -0.834122896194458, 0.36185041069984436, 0.5306025147438049, -1.3177882432937622, -0.833427369594574, 0.18408720195293427, -0.6705369353294373, -1.3102028369903564, -0.11017893999814987, 0.12110480666160583, 0.17832912504673004, -0.8449241518974304, 0.8781489729881287, -1.1580748558044434, 0.32894888520240784, 0.29231154918670654, 0.5841509103775024, 0.49998748302459717, 0.18791913986206055, 1.0482568740844727, -0.20142655074596405, -2.347938060760498, -1.2165727615356445, -0.07699936628341675, 0.8130163550376892, -0.21530069410800934, 1.4866222143173218, -1.7932344675064087, -0.4988101124763489, -1.677711009979248, 0.8652230501174927, 0.12901316583156586, 0.4955154359340668, 1.781576156616211, -0.04196535050868988, 0.29528719186782837, -0.5822341442108154, 0.9944881200790405, -0.4505344033241272, 0.2263409048318863, 0.4626568555831909, 1.4580726623535156, -0.4088761508464813, 1.8046743869781494, 0.785225510597229, 0.10119971632957458, 0.1234983429312706, -1.812278151512146, -0.7431111931800842, -0.6516435146331787, 0.31670132279396057, 0.41619107127189636, -0.7894812226295471, 0.48625245690345764, -0.41940540075302124, -1.9316574335098267, -0.7112947702407837, -0.5655671954154968, 0.6530576944351196, 0.610099732875824, 0.866995096206665, 1.791209101676941, 1.2427335977554321, 0.18072696030139923, -0.10052347183227539, 0.345649778842926, -1.081953525543213, 1.2651864290237427, -0.20223821699619293, 1.1198275089263916, 0.5321115851402283, 1.2231394052505493, -0.7663130164146423, -0.8869788646697998, -1.8475360870361328, 1.0715755224227905, 0.08868878334760666, 0.871361494064331, -0.5793330669403076, -0.1007072925567627, -1.8718897104263306, -0.5583829283714294, 0.6113154888153076, 1.3257864713668823, 2.408284902572632, 0.37401872873306274, 0.44013431668281555, -1.1782249212265015, -0.6447069048881531, -1.703413963317871, -0.46153295040130615, 0.8594403862953186, 1.0504599809646606, 1.039621353149414, 1.4345943927764893, -0.6666055917739868, -1.138066053390503, -1.6090977191925049, 0.7883138656616211, 1.6266974210739136, 1.0569579601287842, 0.7082438468933105, -0.3125959038734436, -0.73418790102005, 1.8484294414520264, 0.14429143071174622, -0.6981722116470337, 0.7866107821464539, 0.7072978019714355, 0.12541532516479492, 1.3803173303604126, -0.2239735722541809, -0.7968968152999878, 1.6905739307403564, -2.078246593475342, -0.9872463941574097, 0.38698723912239075, -1.461427092552185, 0.7032124400138855, -0.20358900725841522, 0.46984511613845825, -0.732413649559021, 0.0007455742452293634, 1.1129579544067383, -0.6911882162094116, 2.621781826019287, -1.7687249183654785, 0.8560400605201721, 0.10683324933052063, 0.7245293259620667, 0.6020730137825012, 0.8812087774276733, -0.932056188583374, -0.1670333594083786, -0.2091251015663147, 0.9867556095123291, -0.6116728782653809, 0.7007594108581543, 0.8176684379577637, -0.0033072512596845627, -0.11294838786125183, -1.453373908996582, 0.3763004541397095, 0.24199533462524414, -0.20502424240112305, 0.23861466348171234, -0.6306799650192261, -0.355105459690094, 1.4700084924697876, 0.8299601674079895, 0.9461447596549988, 0.8605839610099792, -0.4750347137451172, -0.24378950893878937, -1.53317391872406, 0.13350260257720947, 0.9201833009719849, -1.651357889175415, -0.8977619409561157, -0.6215301156044006, 1.2383137941360474, 1.3123098611831665, -0.25270912051200867, 0.9316617250442505, -0.23500941693782806, 0.6784819960594177, 0.39000123739242554, 1.249922275543213, 1.7575454711914062, 0.047995585948228836, -0.7692952752113342, 0.3510785698890686, 0.047291677445173264, 0.052908264100551605, 0.8630212545394897, 0.8547642230987549, 0.10566810518503189, -0.1453629434108734, -0.34576743841171265, 1.7889254093170166, 1.3002853393554688, -0.6472582221031189, -1.7818018198013306, -0.10989569872617722, -0.9507420659065247, 0.31938326358795166, -0.14378589391708374, 0.6884105205535889, -0.711949348449707, 0.0035638227127492428, 0.5204012989997864, 0.35450097918510437, 1.2427616119384766, -0.2742694914340973, 2.821049451828003, 0.18013671040534973, -0.6569302678108215, -0.850732684135437, 1.136033296585083, 1.2279127836227417, 0.8481614589691162, -1.2101287841796875, -0.9578470587730408, 0.5535447001457214, -0.5211922526359558, -1.8473421335220337, 0.35299375653266907, -0.11750348657369614, 0.42757025361061096, 2.560940742492676, 0.10475581139326096, 0.4126584231853485, 0.2557562291622162, -0.1843830794095993, -0.5744773149490356, 0.10226032882928848, 0.5847418904304504, 1.10944664478302, 0.6566806435585022, 1.9433319568634033, 0.764386773109436, 0.286796897649765, -2.7354228496551514, -0.3277398645877838, 0.5907067656517029, -1.18009352684021, -1.6594616174697876, 0.2592592239379883, 0.9434971213340759, -1.0619322061538696, 0.48725625872612, -1.027400016784668, 0.6059423685073853, -0.19853557646274567, 0.6303557753562927, -1.3054345846176147, -0.08429476618766785, 0.38999539613723755, 0.038189370185136795, 0.26661795377731323, -0.271646112203598, -0.5157092809677124, -1.7992274761199951, -0.4780634343624115, 0.57938152551651, 1.5117089748382568, -0.8466061949729919, 1.882712960243225, -0.11304186284542084, -2.3314316272735596, 1.5947682857513428, 0.13411913812160492, -0.7863790988922119, 2.4565677642822266, 0.7046186327934265, 0.8781912922859192, -0.4098704159259796, 0.880492091178894, 0.585617184638977, -0.5066172480583191, 1.1030426025390625, -0.8380014896392822, -2.87357497215271, 0.8987902998924255, -0.5388409495353699, -0.8057148456573486, -1.1863183975219727, 0.5434086918830872, 0.3045191764831543, 0.8269433975219727, -1.948032021522522, -1.173988938331604, 1.5601400136947632, 0.4097023904323578, 0.15282760560512543, -0.7323702573776245, -0.4150063097476959, 0.2485133856534958, -1.342296838760376, 1.2822824716567993, 0.8688839673995972, 0.5067474842071533, 1.3720109462738037, 0.6514325141906738, -0.6353892683982849, 0.5327177047729492, 0.7832231521606445, 0.08347444236278534, 1.9970965385437012, -0.48340263962745667, 0.8990408778190613, 0.7888885140419006, -0.2452131062746048, 1.9410291910171509, -1.0615369081497192, -0.14782503247261047, 0.5025869011878967, 0.50221848487854, 1.3227182626724243, 0.41132232546806335, -0.878231406211853, -0.9797818064689636, -0.5840666890144348, -0.2850489318370819, -0.4518702030181885, -0.4306507408618927, 0.6598882079124451, -0.09151573479175568, 1.1705864667892456, 1.0857571363449097, 1.2666789293289185, -0.5598213076591492, 0.5692354440689087, 0.6877869963645935, 0.8328885436058044, 0.5914000868797302, 2.0675041675567627, -0.1940058320760727, -0.686257541179657, -0.5102867484092712, -1.5671027898788452, 0.8191758394241333, -0.45189762115478516, -0.020342528820037842, 0.7802937030792236, -0.6294525265693665, -1.943136215209961, -0.9908303022384644, 0.9032027125358582, -1.0279550552368164, 1.2088215351104736, -1.245437502861023, -0.7753289937973022, -0.09026198089122772, 1.4749090671539307, 1.1696819067001343, -0.3816709816455841, 2.223119020462036, 1.615161657333374, 0.3152664601802826, 0.4112227261066437, -1.4455105066299438, 0.61419677734375, -0.5824171304702759, 0.751680850982666, -0.4393373429775238, -2.05552339553833, -0.7388797998428345, 0.2394726574420929, 0.14271925389766693, -0.8695600628852844, 0.7982721924781799, -0.5102201700210571, -0.5628144145011902, 1.3919655084609985, -0.22483955323696136, 1.3461228609085083, 0.6445153951644897, 0.17979656159877777, -0.2412911057472229, -1.6018720865249634, -1.4863669872283936, -0.09272531419992447, -2.2380738258361816, -0.5417373776435852, -0.7438355088233948, 0.5705547332763672, -0.3150078356266022, 0.3603063225746155, 1.4950008392333984, 1.1019306182861328, -0.9363265037536621, -0.6207606792449951, -0.8849642872810364, 1.1708766222000122, 0.5238529443740845, -0.8338131308555603, -1.4526599645614624, -0.09823164343833923, 0.30132776498794556, -0.13425949215888977, -0.11355992406606674, -1.0060397386550903, 1.813681960105896, 0.09628698229789734, -0.6460930705070496, 0.46698909997940063, 1.752140760421753, -0.3229790925979614, 0.48008376359939575, -1.041232705116272, 0.13321495056152344, -0.3338111937046051, 0.04458469897508621, 0.6214030981063843, -0.6021636724472046, -0.6225634217262268, 0.7761319875717163, 1.2474794387817383, 0.07240288704633713, 1.0758360624313354, -0.4189367890357971, -0.7819222807884216, 0.3441087305545807, -0.24688087403774261, -2.3335671424865723, 0.678739070892334, -0.8633806109428406, 1.7792112827301025, -0.494113564491272, -0.34418636560440063, 0.15461798012256622, 1.4339118003845215, 0.7921304702758789, 0.1282995194196701, -1.4683640003204346, -1.1696341037750244, -0.8072182536125183, 0.6540335416793823, -0.07091046869754791, 0.44750261306762695, -0.31607532501220703, 0.5785621404647827, 1.7944161891937256, 1.6821688413619995, -0.44458672404289246, 1.4397646188735962, 0.8874654769897461, 1.1949902772903442, -0.8463035821914673, -0.06295943260192871, -0.14336486160755157, -0.3542356491088867, 0.2511875629425049, 0.057568375021219254, -0.032130975276231766, -1.3572964668273926, -0.9389835596084595, -0.5699782371520996, -1.372443675994873, -0.2701522707939148, 0.4228340685367584, 0.6315143704414368, 1.565024495124817, -0.7994313836097717, 1.6472679376602173, -0.4507141709327698, -1.1411607265472412, 0.7477549910545349, -0.19205395877361298, 0.009269259870052338, -0.1838831603527069, 1.868373155593872, -2.3735995292663574, -1.657365322113037, -0.4208925664424896, -0.3128889203071594, 1.3723665475845337, -0.39751043915748596, -0.1009054183959961, -1.1269489526748657, -1.8354158401489258, -0.5928371548652649, 1.906113862991333, 0.05758453533053398, 1.2848773002624512, -0.3598129451274872, 0.2724153697490692, 0.46438708901405334, -1.7152272462844849, 0.3836098313331604, -0.2561454772949219, 0.3663513958454132, -0.7769580483436584, 0.46368345618247986, -2.522118091583252, -0.17389141023159027, -1.1874569654464722, 0.3329547345638275, 0.4326457381248474, -0.520605206489563, -1.3116358518600464, -1.326785922050476, -1.0299679040908813, -0.4945897161960602, -1.0813111066818237, 0.8868547677993774, -0.32158711552619934, 0.8610796332359314, -0.49867796897888184, 0.18754245340824127, 0.539211630821228, -1.554137945175171, -1.4380359649658203, 0.7045916318893433, -0.5537316203117371, -0.7604891061782837, 1.1617608070373535, -0.11118052154779434, 1.6061080694198608, -0.0008075480000115931, 0.45982420444488525, -1.2395135164260864, 0.9015027284622192, -0.5734684467315674, -0.6247808933258057, 1.3938822746276855, 0.9234775304794312], [-1.6538643836975098, -0.9823740124702454, -0.6615317463874817, -0.7993494272232056, -0.34981420636177063, 0.6601657271385193, 0.38751575350761414, -0.5765525698661804, 1.0535374879837036, 0.5228766798973083, 0.19094236195087433, -2.078523874282837, -1.7651630640029907, -0.2026027888059616, 0.7337693572044373, 1.8484110832214355, 0.7788415551185608, 0.5908591151237488, -1.0096112489700317, 0.41100409626960754, -1.4109761714935303, 0.6152961850166321, 0.9305926561355591, 0.06261260062456131, 0.4868585169315338, -1.3653903007507324, 0.3500533998012543, 0.3062008321285248, -0.24116171896457672, -0.508564293384552, 0.16725382208824158, 1.2293645143508911, 0.9051638245582581, 0.3202683925628662, -1.34381103515625, 1.1936787366867065, 0.6164178252220154, -0.7457016706466675, 1.070874571800232, -2.219542980194092, -1.2420828342437744, -1.1754649877548218, -0.5972158312797546, 0.3089403510093689, 1.1668930053710938, 0.7823441028594971, 0.6433541774749756, 0.020878618583083153, 0.32570844888687134, -0.21541567146778107, 0.7913833856582642, 1.7276196479797363, -0.21364369988441467, 0.15163366496562958, 0.7101395726203918, 0.44957566261291504, 0.9944962859153748, -0.7990477085113525, -0.8217210173606873, 1.266645073890686, -0.019355807453393936, -1.1931124925613403, 0.5269924998283386, 0.5317820310592651, 2.9726340770721436, -1.010211706161499, -1.4310733079910278, 0.0416569858789444, 0.14832010865211487, -1.9562941789627075, -0.7136619091033936, 0.9548858404159546, 0.2872702181339264, 0.12643533945083618, -0.5286828279495239, -0.5473528504371643, -0.6629905700683594, -0.026530176401138306, -0.36710190773010254, 0.15708869695663452, -0.307973176240921, -0.2894347906112671, 0.31429824233055115, 1.4417715072631836, 0.8237178325653076, -0.2784314453601837, 0.44281506538391113, 0.08292121440172195, -0.4647930860519409, 0.16928817331790924, 0.48233893513679504, -0.7131862044334412, 0.05968159809708595, -0.22466199100017548, 2.548419237136841, 0.35501208901405334, -0.03195394203066826, 0.3374917805194855, -2.1533257961273193, -1.1355470418930054, -0.8168958425521851, -0.8787297010421753, 0.5591315627098083, 0.5784628987312317, 0.032680924981832504, -0.3975748121738434, 0.10200843960046768, 0.9463205337524414, -0.07501967251300812, 0.0011567637557163835, 2.190338134765625, -0.3115709722042084, 0.21308398246765137, -0.2891841530799866, 2.70239520072937, -0.36537203192710876, -0.09558084607124329, -2.019043207168579, -0.09586810320615768, 2.04179048538208, 1.0872222185134888, -1.6108534336090088, 0.31524014472961426, 1.312530279159546, 0.7488507032394409, 1.0701228380203247, -0.14691758155822754, 0.5780013799667358, -0.5415001511573792, -0.7852425575256348, -0.8685304522514343, 2.3694114685058594, 0.017749136313796043, -1.8334288597106934, 1.1431835889816284, -1.0349398851394653, 0.5379489660263062, -0.488646924495697, 0.4678846001625061, 1.4512789249420166, 0.5802820324897766, -1.048209309577942, 0.40424424409866333, -0.7375569939613342, 1.0224794149398804, -0.03773884102702141, -0.3022543787956238, 1.8345109224319458, -0.09863565117120743, 1.8458806276321411, -0.8892994523048401, -1.3445532321929932, 2.4315192699432373, 0.6676661372184753, -0.5185603499412537, 1.4007657766342163, 2.384874105453491, -0.5334487557411194, 1.6062597036361694, 1.5200444459915161, 0.6020092964172363, -0.8053312301635742, -0.553715169429779, 0.204640194773674, 0.397431343793869, 0.6319937705993652, 1.1286659240722656, 0.2991475760936737, -0.06779491156339645, -1.9247281551361084, 0.5588787794113159, 0.32763317227363586, 0.04360336810350418, -0.3142046332359314, 0.23482976853847504, -0.6033948659896851, 0.7623850703239441, 1.1935580968856812, 0.10669328272342682, -0.32159051299095154, -1.2532079219818115, 1.156144380569458, -0.9495015740394592, 0.6505681872367859, 0.3118823170661926, 0.5181301236152649, 0.3809817135334015, 0.9561570286750793, -0.6268258094787598, 0.4384390115737915, 1.959869384765625, 0.6884610652923584, 0.507932722568512, -1.1592047214508057, 0.10209081321954727, -0.7179999947547913, 1.1349467039108276, 0.3270184099674225, 0.2430959790945053, 0.6711530685424805, -1.6360174417495728, -0.44834235310554504, -1.2797950506210327, -1.2595431804656982, 0.1437869817018509, 0.4226386547088623, -0.7033600807189941, 1.9683791399002075, -1.5378504991531372, -0.03888870030641556, -0.8594078421592712, -0.46315088868141174, 0.19846297800540924, 0.7478052377700806, -1.9054865837097168, -0.6714684367179871, 0.4209253191947937, 0.17550547420978546, 0.0318966880440712, -0.23730547726154327, -0.5928977131843567, 0.864560067653656, 0.3319633901119232, -0.6488174796104431, -0.9132384657859802, -0.1063884049654007, -0.5682076215744019, 1.0278702974319458, 1.228896975517273, -0.32554498314857483, -0.08141476660966873, 2.571476459503174, -0.6703464388847351, 1.0736974477767944, -1.4801260232925415, 0.3054611086845398, 0.4762602150440216, 1.0132665634155273, -0.9225037693977356, -0.6818592548370361, -0.5511771440505981, -0.5445036888122559, 2.26826548576355, 1.3445316553115845, -0.0014525013975799084, 0.5202202796936035, 0.036644767969846725, 0.8803435564041138, -0.4384367763996124, -1.8467785120010376, 0.19967252016067505, -1.3810621500015259, 0.3006318211555481, 0.21298621594905853, -1.3326560258865356, 1.03987455368042, 0.2909642457962036, -0.745956540107727, 1.2096487283706665, 0.3885436952114105, -0.566800594329834, 0.19803740084171295, 0.07012881338596344, -0.01577501744031906, 0.6612133383750916, -1.3019136190414429, -1.511996865272522, 0.018548473715782166, 0.021579699590802193, 0.9340766072273254, 0.7747038006782532, -0.1102447584271431, -1.1046324968338013, 0.42213258147239685, -1.3313456773757935, -0.18571718037128448, 0.240146204829216, 0.6449483633041382, 0.45309025049209595, -0.5127881169319153, -0.16033951938152313, -1.4972081184387207, -0.13356970250606537, 0.07128388434648514, 0.5909940600395203, 1.470285415649414, -0.09997729957103729, -0.08364838361740112, 0.23087909817695618, -0.6386802792549133, 1.3237988948822021, 0.09280616790056229, -1.2529805898666382, 1.5086497068405151, -0.041336070746183395, 0.38237565755844116, -0.5773475170135498, -1.7181819677352905, -1.744083046913147, 0.5931248068809509, -0.9251891374588013, 1.2785636186599731, 0.0999888926744461, 1.2417151927947998, 1.2962095737457275, -1.2128347158432007, -1.2205612659454346, 0.4306681752204895, -0.17934386432170868, 0.5102055668830872, -0.3637583553791046, 0.2004125416278839, 0.2025904655456543, 0.819487988948822, 0.8534272313117981, 0.9445244073867798, -0.3536643385887146, 1.5314521789550781, -0.3111819922924042, 0.04874134436249733, 0.08106549084186554, -1.6365629434585571, 0.9815467000007629, 0.583818793296814, -0.5962833762168884, -1.5517067909240723, 2.059706926345825, 0.5476987361907959, -0.5243687629699707, -1.466779351234436, 0.11003930866718292, -0.7619667053222656, -0.6845037937164307, 0.22949343919754028, 0.5654423832893372, 0.3364521563053131, 0.4890785813331604, -0.4002436399459839, 1.10299813747406, 2.153169870376587, -0.7684715986251831, 0.6157427430152893, -1.0764358043670654, 1.7488175630569458, 0.4198266863822937, -0.9059566259384155, -0.1690492331981659, 0.9707639217376709, 1.2302052974700928, -0.739969789981842, 1.4882501363754272, -0.7535012364387512, -1.69809889793396, 1.7066699266433716, -1.4911165237426758, -0.9107482433319092, 1.2719670534133911, -2.3935160636901855, -1.2050964832305908, 1.6048967838287354, -1.2837640047073364, 0.8487290740013123, 0.8004325032234192, -0.3484545946121216, -1.2811498641967773, -0.7552500367164612, -1.3479909896850586, 0.5628676414489746, -0.9583636522293091, -0.41322559118270874, -0.09689286351203918, -1.0248634815216064, -0.1470998376607895, 0.3866431713104248, 0.20803213119506836, 1.6870795488357544, 0.4292804002761841, -0.9783799648284912, 0.13640357553958893, 1.1042530536651611, -0.6828096508979797, -1.806830644607544, -0.8246782422065735, 1.980827808380127, -0.3260684311389923, 0.9707349538803101, 1.7070024013519287, 1.3019236326217651, 0.16177721321582794, 1.7685322761535645, -0.12196965515613556, 0.2517760694026947, 0.012084024958312511, -1.4480500221252441, 0.42545950412750244, 0.025313856080174446, -0.264903724193573, 0.15462280809879303, 0.5189246535301208, 0.4511452913284302, -0.9247303009033203, -0.11762736737728119, -0.2549818158149719, 0.9102141261100769, -0.05773681029677391, -1.8949490785598755, 0.6418865323066711, 2.9125475883483887, 1.0369884967803955, 1.010158896446228, -0.765820324420929, 0.9636844396591187, -2.108980894088745, 1.1974843740463257, -2.099151611328125, 0.7830256819725037, 0.10244104266166687, 0.08235044777393341, 0.38055703043937683, 1.803715467453003, 0.11971036344766617, -0.39993545413017273, -0.6290847659111023, -0.8190996646881104, 0.0833209827542305, 1.1481128931045532, -0.43209558725357056, -1.1025701761245728, 0.8201993703842163, 0.10902464389801025, 0.12933945655822754, 0.47311508655548096, 0.7115622162818909, 2.038862705230713, 0.2307654768228531, -1.0290712118148804, 0.8597159385681152, 0.6850742697715759, -0.424336701631546, 1.6455305814743042, 0.7402860522270203, -0.2891247570514679, -0.26370951533317566, 0.6692968010902405, 0.8547579646110535, -2.5302844047546387, 0.426716148853302, 0.9870574474334717, 1.6643491983413696, 1.226346731185913, 0.14738573133945465, -1.7413824796676636, 1.3180041313171387, -0.7051389813423157, -0.6039699912071228, 0.012532212771475315, -0.9577009677886963, -0.401436984539032, 1.2968279123306274, -1.3485110998153687, 0.5959945917129517, 0.09123577177524567, -0.5339913368225098, 0.06816178560256958, 3.219651460647583, 0.07999415695667267, 0.5816356539726257, 1.3618040084838867, -0.8530541658401489, 0.332824170589447, -0.7506146430969238, 0.3743877410888672, -0.5054319500923157, 1.0349478721618652, -0.25325050950050354, -0.04317404329776764, 0.8830094933509827, 0.5131662487983704, -0.49726563692092896, 0.6849609613418579, -0.47765296697616577, 2.721547842025757, 0.33423125743865967, 0.08090942353010178, 0.013783537782728672, 0.31460684537887573, 0.888806939125061, -1.1651428937911987, -0.9484753608703613, -1.4493616819381714, 0.19898173213005066, 1.2000329494476318, 2.18192982673645, 0.5362076163291931, -0.49941208958625793, 1.7281734943389893, -0.3515664041042328, -0.5016008019447327, 2.0724868774414062, 0.4714115560054779, 0.5729871392250061, -0.3795519173145294, 0.10490357130765915, 1.5205860137939453, -1.63828706741333, -1.3045775890350342, -0.0505070723593235, 0.4432868957519531, 0.7602807879447937, 0.39366409182548523, -1.7753174304962158, -0.21194680035114288, -0.41464513540267944, -1.038943886756897, 2.7722156047821045, -0.00946704763919115, 0.29029372334480286, 3.0073351860046387, -0.3499898612499237, 1.8820219039916992, -0.8309260606765747, -0.2631015181541443, 0.4676404595375061, 0.7327694296836853, 0.21884068846702576, 1.1184265613555908, 0.3421570956707001, -1.3343781232833862, -2.2346925735473633, -0.08925753086805344, -0.4815542697906494, 0.29658105969429016, -0.3085111677646637, -0.1713426560163498, 1.8728078603744507, 0.10435272753238678, -1.327470064163208, 0.5228727459907532, -0.6715682744979858, 1.6410752534866333, -1.2774215936660767, -0.32438066601753235, -0.09762351959943771, 0.8045591115951538, -1.4156510829925537, 0.7757385969161987, 0.6130790710449219, -1.1155924797058105, -0.04569614678621292, -0.41783174872398376, 0.6328167915344238, 0.44204214215278625, -0.4719836115837097, 0.31172865629196167, 0.8511568903923035, 0.57220458984375, 0.18211521208286285, -1.4421416521072388, 0.2174110859632492, 0.7496497631072998, 1.3453327417373657, 0.6405181884765625, -0.4319145977497101, 1.5676525831222534, 0.02583165653049946, -2.793428659439087, -0.22159545123577118, -0.023037320002913475, 0.11537370085716248, 0.12824487686157227, 0.855038583278656, 0.7697919011116028, 1.7648495435714722, 0.2990270256996155, -0.5874214172363281, -0.4511739909648895, 1.3007515668869019, -1.1453359127044678, 0.5772070288658142, 0.676169216632843, -1.1700817346572876, -0.9907656311988831, 0.9697848558425903, -0.8267669677734375, -0.8204546570777893, 0.5120421648025513, 0.0006121799233369529, -1.1287052631378174, 0.12020427733659744, 0.08382754772901535, 0.009726373478770256, -1.04753577709198, -0.6470791697502136, 0.4510175883769989, -0.9936126470565796, -0.5484554171562195, -1.609083652496338, 0.46551913022994995, 0.5411106944084167, -0.845317542552948, 0.4527033865451813, 1.7333035469055176, -1.1628412008285522, -1.4880694150924683, 0.6630895733833313, 0.3821633458137512, 0.1720418930053711, -0.09108218550682068, -0.8965884447097778, 2.042865037918091, -0.44276154041290283, -0.007647423073649406, -2.149506092071533, -1.8693084716796875, -0.15591922402381897, -1.295378565788269, 0.5508909821510315, -0.559846818447113, 0.5213585495948792, 0.7797450423240662, 1.4064058065414429, -0.8876264095306396, -0.06667744368314743, -0.11652622371912003, 0.19334721565246582, -1.0448983907699585, 1.9126653671264648, -0.13622060418128967, -0.8617532849311829, -0.17149090766906738, -0.3382679522037506, 1.8341935873031616, 0.20080098509788513, -1.2229481935501099, 0.24580249190330505, -1.5869863033294678, 1.2957483530044556, -0.5115814805030823, -0.1809646040201187, 1.010037899017334, 0.6201145052909851, -1.1283777952194214, -0.660910964012146, -0.9903422594070435, -0.3716154992580414, -0.33784109354019165, -1.4280239343643188, -1.3003356456756592, -0.8395877480506897, 0.29181188344955444, -0.4280085861682892, -1.2921907901763916, -0.3494859039783478, -0.4552468955516815, 1.655080795288086, 1.2479170560836792, -0.4872640073299408, -1.1166573762893677, 1.937530279159546, 0.3518384099006653, 1.1386027336120605, 0.6614293456077576, 0.8342673778533936, 0.6882738471031189, -1.4382543563842773, -0.6307331919670105, 0.08987581729888916, -0.781819760799408, 0.7512657642364502, 1.8925477266311646, -0.06966212391853333, 1.1556082963943481, 0.7978042364120483, 0.7449493408203125, 0.9797477126121521, 2.5385332107543945, -1.8278921842575073, -0.8455893993377686, -0.4925534725189209, 0.4037253260612488, -0.12479528039693832, 0.3481329083442688, 0.38295021653175354, 0.6020916104316711, -0.0502164401113987, -0.10524173825979233, -0.6298835277557373, 0.3831334114074707, 0.3201029896736145, -1.7474539279937744, -0.6761538982391357, 0.4819260537624359, -2.5441009998321533, 0.4439699649810791, 0.2646329402923584, 0.23851607739925385, 0.3754013180732727, -0.7132019400596619, -0.2476324737071991, -1.037188172340393, 0.050069913268089294, -0.4803057312965393, -0.7937585711479187, -0.43030285835266113, 1.4916335344314575, 0.1849435716867447, 0.39743322134017944, -0.27491623163223267, 1.032080054283142, 1.739748477935791, 2.047419548034668, 0.09528797119855881, 1.0237674713134766, -0.20766408741474152, 0.5783218145370483, -1.2448844909667969, -1.556388020515442, -0.9511496424674988, 1.2348380088806152, -0.34060925245285034, -1.6655141115188599, -1.6654301881790161, -0.6650100350379944, -1.8691273927688599, 1.930986762046814, 0.6532748341560364, -0.4715222716331482, -0.513237714767456, 0.6708455085754395, -0.5535642504692078, -1.2469096183776855, 0.9381719827651978, 1.5861085653305054, 0.6207049489021301, -0.46241962909698486, -0.33839619159698486, 1.357529878616333, -0.14269275963306427, 0.30422285199165344, -0.2506569027900696, -0.019592715427279472, 0.1264672577381134, -0.06333307921886444, -2.4694066047668457, 0.3010946810245514, 0.004484976176172495, 1.293607473373413, -0.3389900028705597, 0.7287603616714478, -1.104382038116455, -1.3029520511627197, -0.4376106858253479, 0.9765419363975525, -0.43020936846733093, 0.48030489683151245, -0.19798538088798523, -1.4351906776428223, -0.7768720984458923, -0.326884001493454, -2.6975111961364746, 0.41875991225242615, 0.40102648735046387, -1.6150275468826294, -0.7095770239830017, 1.670069932937622, -0.9272310137748718, -0.27814948558807373, -0.42286020517349243, -1.2333585023880005, 0.22439488768577576, 1.1657689809799194, 1.1361407041549683, -0.19919204711914062, -0.143667533993721, 1.2616969347000122, -0.33932366967201233, -1.227245807647705, 0.7761366367340088, 1.0991277694702148, 1.8097937107086182, -1.3725576400756836, -0.0714748352766037, 0.5498318076133728, -0.28426867723464966, 1.801710605621338, 0.7633165121078491, 0.05574895069003105, -0.4157343804836273, -0.6647492051124573, -2.060431718826294, -1.068429946899414, 0.5659272074699402, 0.2671201527118683, -0.14331582188606262, -0.7311822772026062, 0.941555380821228, -1.6357905864715576, -0.25657445192337036, -2.5308918952941895, 0.8271831274032593, -2.1007027626037598, 1.6806066036224365, 0.0038375286385416985, -2.2285714149475098, 0.9190735220909119, 0.85555499792099, -1.5938968658447266, -0.08916110545396805, -1.177512526512146, -0.5042868852615356, 0.33483538031578064, 0.04405594617128372, -0.7042215466499329, -0.3191128373146057, 0.20814716815948486, 1.0682107210159302, -1.2264777421951294, -0.19834868609905243, 2.252277135848999, -1.5438345670700073, -0.05169299244880676, 0.4223588705062866, 0.5196835398674011, 0.12089772522449493, -1.2872769832611084, 2.1947693824768066, -1.0187617540359497, 0.1615724414587021, 0.2874651253223419, -1.359681248664856, 2.3246049880981445, 1.262445330619812, -2.759047746658325, 0.031203050166368484, 1.738452672958374, 0.016215816140174866, 1.6051290035247803, -0.0514739453792572, 0.5015496015548706, 0.7743121981620789, 0.7903977036476135, 0.7961564064025879, 0.7244012355804443, 0.06790709495544434, 0.7491557002067566, 0.7918661236763, -1.9446544647216797, 1.4653724431991577, 0.4319555163383484, -0.650470495223999, 1.1491223573684692, 1.3347519636154175, -1.5394924879074097, 0.3844851553440094, -1.1264309883117676, 0.5184287428855896, 0.8790470361709595, -0.3643151819705963, -0.42117416858673096, 1.118276596069336, -0.8382527232170105, -0.32284072041511536, 2.223525285720825, -0.39455658197402954, -0.44963452219963074, -0.7861881852149963, 0.17871256172657013, -0.34060969948768616, 0.9245838522911072, -0.488063782453537, 1.2313796281814575, 0.5375614166259766, 0.2899530827999115, -1.261012315750122, 0.7781315445899963, -0.2583784759044647, 1.2012457847595215, 1.6133027076721191, -0.7228275537490845, -1.0939444303512573, -1.523626685142517, -0.714779794216156, -2.3248965740203857, 0.5024275779724121, -0.1472913920879364, -0.5005150437355042, 0.13594992458820343, 0.3782222867012024, -0.7813911437988281, 0.42041558027267456, 1.9346842765808105, 0.49373897910118103, 0.6025072336196899, -0.47941693663597107, -1.1380561590194702, 2.113887071609497, -1.0228092670440674, 1.189066767692566, 0.2817167639732361, -0.1271757334470749, -0.8857730031013489, -2.367152452468872, -0.9126392602920532, -0.015851622447371483, -0.16683822870254517, 1.6028027534484863, 0.03541199490427971, 0.8459303379058838, 0.24519449472427368, 0.35765159130096436, 0.4021359384059906, 1.0198229551315308, -1.624254822731018, 0.434501975774765, 0.9543294310569763, -0.520334005355835, -2.17448353767395, 0.8741068243980408, 1.0933130979537964, -0.4219590723514557, -0.25465407967567444, 0.3308652937412262, -0.4716586470603943, 0.8949382901191711, 0.49798905849456787, 0.10481198132038116, -0.395735502243042, -1.2320955991744995, 1.5593500137329102, -2.236907720565796, 0.21390578150749207, 0.724229097366333, -1.5987120866775513, -0.7674133777618408, -0.7420034408569336, 0.2146054208278656, -0.1485058069229126, 0.628352165222168, 0.06131143867969513, -2.366713762283325, -0.31620684266090393, -0.8268271684646606, -0.2452215552330017, 0.35535353422164917, -1.102831244468689, 0.3642893433570862, -1.7361681461334229, -0.3570871651172638, 1.3628727197647095, -1.2374980449676514, 0.8683987855911255, 0.04746541380882263, -1.4846301078796387, -2.6970102787017822, 0.3696829378604889, 0.9466128349304199, -0.0053583066910505295, -0.17119626700878143, -0.12784001231193542, 0.19102074205875397, 1.0090703964233398, 1.103619933128357, -2.6646058559417725, -0.6813144087791443, 0.9063739776611328, -0.6571633815765381, 0.22727221250534058, -0.21332819759845734, -1.1365852355957031, -0.4420403838157654, 0.41638365387916565, -0.18231162428855896, 0.5520501732826233, 0.13167405128479004, 0.5933130383491516, -0.5987493991851807, 0.5943567156791687, 0.4341578781604767, 0.8397496938705444, 0.5177902579307556, 1.8298298120498657, 0.853614866733551, 0.20253050327301025, -1.1120424270629883, 0.3065047860145569, -1.0052763223648071, 1.8704674243927002, 0.7482219338417053, 0.5099040269851685, -0.9529717564582825, -0.9616838693618774, 0.34451404213905334, -0.8137474060058594, -1.6807948350906372, 2.6642043590545654, 0.44285547733306885, 0.049565721303224564, 0.9658304452896118, 2.1400816440582275, 0.6185892224311829, 0.20730002224445343, -0.925996720790863, 1.4214602708816528, -1.4099174737930298, -1.0743588209152222, -0.44651666283607483, 1.6301050186157227, 0.4445714056491852, -0.4119415879249573, 0.4404568076133728, -0.0077673872001469135, 0.5383027791976929, 0.2516302764415741, 0.47650015354156494, 0.08784662187099457, 0.6588800549507141, -2.3701977729797363, -0.7560104131698608, 1.1099578142166138, -1.958308219909668, 1.2554340362548828, -0.6004329323768616, -0.6058332324028015, 1.0964820384979248, 0.03366619348526001, -1.3533284664154053, 0.42446795105934143, -1.4833494424819946, 1.6702214479446411, -0.6245384216308594, -0.061731815338134766, -0.35795122385025024, -1.2838354110717773, 0.2183259129524231, 1.0727437734603882, -1.2279552221298218, -0.10749107599258423, -0.047303009778261185, -0.9186905026435852, 1.2110321521759033, 0.3198634386062622, 0.7036409378051758, -0.0909779742360115, 0.06543515622615814, -0.7431457042694092, -0.7231736779212952, 0.5756445527076721, -1.0811386108398438, 0.7390190958976746, 0.7444355487823486, 1.6199777126312256, -0.3920022249221802, -0.4242016673088074, 0.0673912912607193, 0.22411248087882996, 0.7428638339042664, 0.7640950083732605, 0.43237191438674927, 0.6292909383773804, -0.6763704419136047, 0.10694455355405807, -0.2328312247991562, -0.7600440382957458, -1.3044432401657104, 1.0866831541061401, -0.599026620388031, -0.21067379415035248, -0.8814444541931152, 1.378126859664917, 0.823638916015625, 0.13810209929943085, 0.14191603660583496, 0.4228767454624176, -0.4622879922389984, -0.4595290720462799, 0.5663084983825684, -1.008042812347412, 0.18990559875965118, -0.49225515127182007, 0.9650548696517944, -1.5612385272979736, 0.6439874768257141, 2.2995173931121826, 2.037158966064453, -0.7424512505531311, -0.453053742647171, -0.7181513905525208, -2.0989673137664795, 1.12275230884552, -2.2935781478881836, 0.7995216846466064, 1.3410954475402832, -1.810394525527954, 2.12444806098938, -0.20281684398651123, 0.3160330355167389, -1.717909336090088, -0.612777829170227, 0.8501006960868835, -0.35131460428237915, 0.42549237608909607, -0.27306893467903137, -1.1664149761199951, 0.9512466788291931, 0.9841875433921814, -0.04730432108044624, -0.036394037306308746, 1.468973994255066, -1.1525070667266846, 0.11658991128206253, 0.03825113922357559, -0.5529590845108032, 0.6784981489181519, -1.4798898696899414, -1.1702241897583008, -1.1104718446731567, -1.1291282176971436, -0.213717982172966, -0.24376195669174194, -1.1188499927520752, 0.8009458184242249, -1.35165274143219, 0.5350714921951294, 0.9298692345619202, -1.1612814664840698, 1.3860012292861938, -0.9984409809112549, -1.8336989879608154, -0.28738245368003845, 2.5074729919433594, 0.5145783424377441, 1.2581652402877808, -0.9560571312904358, -0.40983930230140686, -0.18995870649814606, 0.6374263763427734, -1.0619240999221802, 0.4251469373703003, -1.5624680519104004, -0.1560010313987732, -1.2354161739349365, 0.0707995817065239, 0.810103714466095, -0.3355819880962372, -1.8697636127471924, -0.8838454484939575, -0.29305773973464966, -0.6967601180076599, -1.089658260345459, -2.213809013366699, -1.2834564447402954, 0.5721124410629272, -0.5767284631729126, 1.3374333381652832, 1.6650415658950806, -0.05888769030570984, 0.18337832391262054, -2.932182788848877, 1.4764138460159302, -1.7335606813430786, -0.7532567381858826, -1.0292457342147827, 0.6051138043403625, -0.16812223196029663, 0.38269659876823425, -0.27398499846458435, 0.24771663546562195, -0.6046319603919983, 0.8130512833595276, 0.169368714094162, 1.0532150268554688, -1.3420734405517578, 0.48589417338371277, -0.18882615864276886, 0.026453984901309013, -0.12816324830055237, -2.0077598094940186, 0.6698335409164429, 1.304537296295166, 0.4676772654056549, -0.8441299796104431, 2.0120849609375, 2.1726741790771484, 1.1942387819290161, 0.5762672424316406, -0.6837377548217773, 0.36665138602256775, -0.29617899656295776, 0.01395538728684187, 1.0663882493972778, -0.5382903218269348, -0.40283331274986267, -0.10746629536151886, 0.4720676839351654, 0.4060032367706299, 0.33195769786834717, 0.9504659175872803, 0.5731425881385803, -1.3274190425872803, -0.11811883747577667, -0.6325190663337708, -0.4432066082954407, 0.2574124336242676, 0.18850834667682648, 0.44282975792884827, 0.30272427201271057, -0.3355471193790436, -0.2071872353553772, -0.7673370838165283, 0.5931894779205322, -0.44090017676353455, 1.059967279434204, -0.4959152936935425, 1.6935679912567139, 0.18888472020626068, 0.11773759126663208, 0.35759827494621277, -1.635564923286438, 1.2381943464279175, -0.9596572518348694, -1.2217285633087158, -0.6832711696624756, 0.2872706651687622, 0.503902792930603, -1.7626618146896362, -0.730407178401947, -1.2635834217071533, 1.3808379173278809, 0.044822558760643005, -1.9841995239257812, 2.306107521057129, 0.47291338443756104, -0.4887399971485138, -0.9419870972633362, 0.35954612493515015, 0.7819880843162537, -0.22413094341754913, -0.7287960648536682, -0.37283873558044434, -0.4822312593460083, -1.0215542316436768, -0.685015082359314, -1.4494318962097168, -0.8601623177528381, -1.1250377893447876, 1.9333841800689697, 0.16877543926239014, 1.5666999816894531, -1.1181401014328003, -1.5374659299850464, -0.07508603483438492, -1.023585319519043, -0.09183492511510849, 1.1471284627914429, -0.5729483366012573, -0.952900230884552, -1.0984443426132202, -0.37826302647590637, 0.2374720424413681, 0.13947831094264984, -0.9896101355552673, -0.6570925712585449, 0.2569049000740051, 0.2875441312789917, 1.5509610176086426, 1.2483505010604858, -2.478618860244751, 1.7703112363815308, -0.09021053463220596, 0.03435586020350456, 0.02879534661769867, -0.7935852408409119, 0.4122476875782013, -0.26435187458992004, -1.7272393703460693, -0.7401500344276428, 0.42681482434272766, -0.2192825973033905, -1.2438782453536987, 1.1392773389816284, -0.8307856321334839, 0.37882736325263977, 0.9878193140029907, 1.6780924797058105, -1.448691487312317, 0.2474018782377243, -2.144125461578369, -0.544016420841217, 1.482707142829895, -1.1636871099472046, -1.7205580472946167, 1.3535267114639282, -0.10853607952594757, 2.3558177947998047, -1.4788707494735718, 1.0066102743148804, 0.7357586026191711, 0.8110366463661194, -0.9974201917648315, 0.10393773764371872, 1.9856789112091064, -0.2762104570865631, -0.507872462272644, -0.5532292127609253, 1.691470980644226, -0.08949718624353409, -1.4176909923553467, -0.5890418291091919, -0.7698853611946106, 1.9405322074890137, 0.1806398183107376, 1.2788655757904053, 1.9793068170547485, -1.1998649835586548, 0.09977508336305618, 0.3369005620479584, -1.6127402782440186, 0.5435662865638733, 0.21063804626464844, -0.333936870098114, -0.08178164809942245, -0.14017312228679657, 1.1088840961456299, -0.6481457948684692, 1.0331172943115234, -0.5068256855010986, -0.057864464819431305, -1.3931115865707397, 0.3016177713871002, 0.9121476411819458, 0.2192666232585907, 1.0372397899627686, 0.7131474018096924, -0.503931999206543, 1.1499595642089844, -0.7655905485153198, 0.5321899056434631, -0.3322223126888275, -2.1883928775787354, -0.3050522208213806, -1.1813393831253052, 0.4375733733177185, 0.08238855004310608, -1.05427885055542, -0.8529103398323059, -0.6216145753860474, -0.6278873085975647, -0.08011482656002045, 0.049180563539266586, 0.9673361778259277, 0.8512051105499268, 1.3877816200256348, -0.37461209297180176, -0.9247222542762756, 0.7982683181762695, -0.5814002752304077, 0.930457353591919, -0.16421286761760712, 0.15094377100467682, 0.3261052370071411, 0.8132672905921936, -0.5950978398323059, 0.23637200891971588, 0.8502740263938904, 0.14270232617855072, -1.532871961593628, -0.2739158570766449, 0.855061948299408, 0.0006873942911624908, 0.16753633320331573, 1.3110771179199219, -1.8722633123397827, 1.6322579383850098, 1.45534348487854, -0.5585430264472961, 1.1622400283813477, -0.16737546026706696, 0.7133804559707642, 1.4272276163101196, 0.8011273741722107, 0.857893168926239, -0.23465333878993988, -1.3761804103851318, 0.6861809492111206, 0.5566856861114502, -2.1367690563201904, 0.2977176606655121, 1.456940770149231, -1.5250647068023682, -1.3151944875717163, 1.1112371683120728, -1.501356840133667, 1.1311603784561157, 0.3742678463459015, 0.5812395811080933, 1.3600363731384277, 0.6235473155975342, -0.9508267641067505, -1.1007719039916992, 0.5381564497947693, -0.74607253074646, -0.09085754305124283, 1.0485565662384033, 0.3733580708503723, -0.5263165235519409, -1.2185823917388916, -0.34104013442993164, 0.7991513013839722, 0.7754508256912231, 1.4242509603500366, -0.35099104046821594, -0.13645915687084198, -1.5705264806747437, -2.0074188709259033, 1.1136951446533203, -1.187425971031189, 0.6971712708473206, -0.4662962257862091, -0.8746540546417236, -1.7971795797348022, -0.76604163646698, -0.3465992510318756, 0.5455358028411865, 0.08936896175146103, -1.3866714239120483, 1.1926804780960083, 1.9691665172576904, 0.4087413549423218, 0.7402219176292419, -0.35707002878189087, -0.6571367979049683, 0.2926102578639984, -0.3197627365589142, 0.12539425492286682, 0.29944509267807007, -1.46857750415802, -0.5329422354698181, -0.8609406352043152, 0.642719566822052, 0.30541375279426575, -0.5309717059135437, 0.8235697150230408, 0.8166223764419556, 0.19151277840137482, 0.8645927906036377, -0.28617069125175476, 1.8567110300064087, -1.5287249088287354, 0.8454818725585938, -1.0257046222686768, -1.3118767738342285, -0.02496749721467495, 0.8888658881187439, -0.6937330365180969, 2.1566479206085205, 1.6673744916915894, -1.6323630809783936, 0.479374498128891, -0.37377262115478516, 0.22323691844940186, 0.9499144554138184, -0.8566311001777649, 0.11802053451538086, 0.30842679738998413, -0.7166323661804199, 0.9954843521118164, 0.6080865859985352, 0.26535123586654663, 1.1673088073730469, 0.3459475338459015, 1.3334709405899048, -1.401983380317688, 0.22955277562141418, 1.4587470293045044, -0.5826992392539978, 3.5705435276031494, 2.3896918296813965, 0.6278117299079895, 0.7675363421440125, 0.7811732292175293, -1.0842634439468384, 1.1259225606918335, -0.14297905564308167, -0.5653645396232605, -0.6106708645820618, 1.0186790227890015, -0.2943287491798401, -1.3828589916229248, 0.1665409356355667, 1.0646989345550537, 0.3482627868652344, 0.9631077647209167, 0.47032150626182556, 1.0696123838424683, 0.5596938133239746, -0.43244290351867676, 0.9922349452972412, -0.5946398973464966, -1.6297622919082642, 1.7273603677749634, 0.6116682887077332, 0.33738240599632263, -0.9970698356628418, 1.0525637865066528, 0.19733574986457825, 1.3621063232421875, -0.8700352907180786, 1.5982757806777954, 2.108074188232422, 0.07633057236671448, -0.5500308871269226, 0.38858696818351746, 0.7045087218284607, -0.3521275520324707, -0.5992918610572815, -1.3806064128875732, 1.060279130935669, 0.2572272717952728, 0.4057917296886444, 0.13867291808128357, -0.4363775849342346, -1.1847647428512573, -0.9423063397407532, 1.0685054063796997, 0.08911032974720001, -0.0029343299102038145, 0.20203673839569092, 2.3017966747283936, -0.39955246448516846, 0.24099381268024445, 0.1846579909324646, 0.867332398891449, 0.17216254770755768, 0.6885272860527039, 1.0106396675109863, 1.442299485206604, -0.7815254926681519, -1.2667875289916992, 1.326014518737793, -0.8008877635002136, -0.14522765576839447, -0.6560916304588318, -0.23022741079330444, 0.11046914011240005, 0.5728957653045654, -0.14680759608745575, 0.31330418586730957, 0.8694987297058105, -0.6573293209075928, -1.672644019126892, 0.676163911819458, -0.5671782493591309, 0.9857619404792786, -1.702699899673462, 1.4232114553451538, -1.9155868291854858, 0.283112108707428, 0.18329806625843048, 1.259398341178894, -0.3034574091434479, -0.0056032463908195496, 0.7572387456893921, 0.7703706622123718, -0.5804129242897034, 0.6431179642677307, -0.5530421733856201, -1.2582049369812012, -0.5040290355682373, 0.37636634707450867, 1.082045316696167, -0.584330141544342, -0.6432622671127319, -1.8355493545532227, -1.743906855583191, 0.22256691753864288, -0.12882757186889648, 0.32344019412994385, -1.475770354270935, 0.25820112228393555, 0.14167767763137817, 1.95133638381958, 0.013975611887872219, 0.19575349986553192, -1.6736934185028076, -0.786591649055481, 1.1044938564300537, 1.1761906147003174, -1.1232976913452148, -0.3366868197917938, 1.329648494720459, -0.1607617884874344, 0.9643476009368896, -0.5956314206123352, -0.03683317080140114, 0.6500340700149536, -0.8229281306266785, 1.1444191932678223, -1.364263653755188, -0.5911816954612732, -0.7079572081565857, -0.25655823945999146, 1.149335265159607, 0.519832968711853, -0.3463892340660095, -1.3628363609313965, 0.17262403666973114, -0.17969389259815216, 0.2684969902038574, -1.1800869703292847, -1.720597505569458, -0.18762870132923126, -0.8571088314056396, 1.0046272277832031, 1.1164445877075195, 0.34226253628730774, -0.6545482873916626, 1.5159703493118286, 0.3943629860877991, 0.006275614257901907, -0.48827528953552246, -1.283461093902588, 0.3626531660556793, -0.46857601404190063, 1.411157250404358, -1.1826595067977905, -0.8697662949562073, 0.24354587495326996, 0.43245378136634827, -0.2657473683357239, -0.18066659569740295, 0.2926311492919922, 0.8073023557662964, -0.5430466532707214, 0.1495177298784256, 0.34750112891197205, 0.6238448023796082, 0.7202200293540955, 1.245059609413147, -2.6215405464172363, -0.37525680661201477, -1.8902952671051025, 0.6853440999984741, -0.053867653012275696, 2.1623542308807373, -0.13247796893119812, 2.1416666507720947, 0.5313370227813721, -0.8987024426460266, 1.7013342380523682, -1.0811693668365479, -0.8836498856544495, 1.637123703956604, 1.1747334003448486, 0.7260277271270752, 0.324937105178833, 0.35849493741989136, 2.70297908782959, -0.42050406336784363, -0.997655987739563, -0.830032229423523, 0.0008440056117251515, -1.199244499206543, -0.4017797112464905, -0.9233314394950867, -0.0421140156686306, 0.36281079053878784, -0.16472965478897095, -0.9049415588378906, 1.2181156873703003, -0.7509230971336365, 0.6227213144302368, 1.2675284147262573, 0.8666908144950867, 0.9236822128295898, -3.275254964828491, -0.28927192091941833, 0.9795678853988647, 0.972613513469696, 0.727971613407135, -0.42499953508377075, 0.9533737897872925, -0.23541289567947388, -0.9432234764099121, -0.37291261553764343, 0.6187314391136169, 1.266566276550293, 0.012601088732481003, -0.8870358467102051, 0.9542021155357361, -2.255444288253784, 0.7023069262504578, -0.9585195779800415, -1.1865906715393066, 0.5655137896537781, 1.9643657207489014, -1.2367287874221802, -0.6247007846832275, 1.6899514198303223, 1.6213666200637817, 0.6342482566833496, 0.34216544032096863, 0.007256831042468548, 1.512768030166626, -0.9617190361022949, 0.345441073179245, 0.24944335222244263, -1.0186420679092407, 0.826331615447998, 1.05241060256958, -0.8520194292068481, 0.822792649269104, 0.7275882959365845, 0.6559571623802185, -0.9035018086433411, 2.2098350524902344, -0.3521963059902191, -0.0658271536231041, 1.5681231021881104, 0.928115963935852, 1.2640063762664795, 1.1725451946258545, 0.27729931473731995, -2.3346211910247803, -0.5958517789840698, 1.699694037437439, -1.1413780450820923, -1.5657354593276978, 0.1734660416841507, -0.6866975426673889, -0.013079574331641197, -1.4037110805511475, -0.4264219403266907, 1.1884915828704834, -0.4846280515193939, -0.9369681477546692, -0.1635603904724121, 1.8076636791229248, 0.3367435336112976, -0.5034337639808655, 0.30844810605049133, -0.8994214534759521, 2.3847250938415527, 0.5209726095199585, -0.4377478063106537, -0.3422029912471771, 1.1339802742004395, -1.387296438217163, 1.4031215906143188, 0.24991381168365479, -0.2792735695838928, -0.10582853853702545, -0.7429680228233337, 0.4419397711753845, -0.6965311765670776, -0.5019663572311401, -1.6697044372558594, 1.1406234502792358, 0.791750967502594, -2.0654454231262207, -0.22611340880393982, 0.4960362911224365, 0.20006005465984344, 0.6774019002914429, 0.2160903811454773, 0.1414470225572586, 0.7155728340148926, -0.8492647409439087, 0.5062811970710754, 0.40842586755752563, -0.761893093585968, -0.7903510332107544, -0.5248587131500244, 0.04798556864261627, 0.723892331123352, -0.016794955357909203, -0.675598680973053, -0.911442756652832, -0.2699846923351288, 0.14365310966968536, -0.7507765293121338, 1.5655142068862915, 0.8620423674583435, -0.7269850969314575, -0.6407551765441895, -1.2910798788070679, 0.18998941779136658, -1.278100609779358, -0.8604611158370972, 0.33107972145080566, 0.7392273545265198, -0.508489191532135, -1.0088685750961304, 1.388444185256958, -0.5414100289344788, -0.6084070801734924, -0.2603176534175873, -1.6651558876037598, -0.2928069233894348, 1.2503644227981567, -0.48407694697380066, 1.5663591623306274, -0.28074005246162415, -1.2013447284698486, -0.8981360197067261, -1.1686569452285767, -0.5622941255569458, -0.3635502755641937, 0.7298555374145508, -0.4672175347805023, -0.6630979180335999, 0.8204213976860046, 0.9623452425003052, 0.5207721590995789, -0.14541740715503693, 0.5886862277984619, -0.34167274832725525, 0.1163628101348877, -1.0145869255065918, 0.38751980662345886, -3.6707816123962402, -1.0319267511367798, 0.23312020301818848, -1.6998426914215088, -0.13810783624649048, 0.2612229883670807, 1.9299092292785645, -1.2217864990234375, -0.06618531793355942, -1.2278928756713867, -0.12323044240474701, 0.17963393032550812, -1.0148437023162842, -1.6172785758972168, -0.5691388249397278, -1.5317771434783936, 0.5116125345230103, 0.033092647790908813, 1.2986427545547485, 1.850773572921753, 0.8926293253898621, 0.09879060089588165, 0.2274007499217987, -1.0189261436462402, 1.2844380140304565, -0.252213716506958, -0.8328242897987366, -0.5947084426879883, -1.025831937789917, 0.9730806350708008, -0.053077343851327896, 1.2598381042480469, -0.3103211522102356, -0.6142451763153076, 0.6042863726615906, -2.1806986331939697, 1.0215510129928589, 0.31326451897621155, -0.2693750560283661, -0.528179407119751, 0.045828718692064285, 0.12490525841712952, -0.5391053557395935, -0.16664329171180725, 0.36480510234832764, -0.5420857667922974, 0.07661819458007812, -0.14316219091415405, 0.22553160786628723, 2.2256968021392822, 0.2251846194267273, 1.3271652460098267, -1.0389659404754639, -0.31116026639938354, 1.90933358669281, 0.2737767994403839, -1.1725397109985352, 1.0015554428100586, 0.31865194439888, 0.3956725001335144, -0.28282350301742554, 1.3630985021591187, -0.3497680127620697, 0.5749929547309875, -0.5838311314582825, 1.7660235166549683, -1.7384452819824219, -0.834300696849823, -0.5775188207626343, -2.1637675762176514, -0.6906282305717468, -0.88728266954422, -0.786209762096405, -1.13396418094635, 0.3416062295436859, 1.6047110557556152, 0.7365541458129883, -0.28819891810417175, -0.1265878975391388, -0.9303985238075256, -0.28243234753608704, 0.13601037859916687, 0.33212971687316895, 0.025439806282520294, 0.8807288408279419, 1.1426303386688232, 0.22230654954910278, 0.0883016511797905, -0.585075318813324, -0.5729696154594421, -0.3243162930011749, -0.6134111881256104, -1.8846715688705444, -1.7318159341812134, 0.1108798161149025, 0.12626849114894867, 0.7449911832809448, 1.3476375341415405, -0.9069976210594177, -0.9167097210884094, 0.7769861817359924, 2.3697497844696045, 0.15963003039360046, -0.4762164354324341, 0.21768426895141602, -0.2946751117706299, -0.2600981593132019, 0.38152748346328735, 0.10886673629283905, -0.06569540500640869, 0.08546141535043716, -0.016146739944815636, -0.195421040058136, -0.48425158858299255, 0.4734944999217987, 1.1227837800979614, -1.9508593082427979, -0.6364840865135193, -0.4594547748565674, -1.1050580739974976, 0.5419176816940308, 0.518070638179779, -0.31951141357421875, -1.2047083377838135, -1.9651259183883667, 0.9962477087974548, -0.4568410813808441, 0.463317334651947, 0.08228960633277893, 2.3761017322540283, 1.2985119819641113, 0.3904673755168915, -0.3162853717803955, 0.14325262606143951, -1.0426898002624512, 0.6038343906402588, -0.11594856530427933, -0.6671773195266724, -1.3735477924346924, 0.15208949148654938, -0.7596818804740906, 0.7916314601898193, 2.0497097969055176, -0.9640951752662659, 0.16248610615730286, -0.2892709970474243, -1.3997461795806885, -1.3290929794311523, -0.3061637580394745, 1.2577491998672485, -0.6268003582954407, 0.11159665137529373, -0.30997610092163086, -1.8894480466842651, -0.5114244222640991, 0.5003034472465515, -0.7769646048545837, -0.3469441533088684, -0.3990933895111084, -1.9861903190612793, -0.26489248871803284, 1.2693694829940796, 1.4585198163986206, -0.6786648035049438, -1.0683484077453613, 0.36454126238822937, -0.41259920597076416, 0.3589872419834137, 1.1220062971115112, 1.6102228164672852, -0.2079378217458725, -0.6766006946563721, -0.47646233439445496, 0.9763368964195251, -0.31457141041755676, 0.31934866309165955, 0.6150150895118713, 0.4707294702529907, 0.28068864345550537, 1.8997248411178589, 0.6843909621238708, 0.00015352421905845404, 1.3324272632598877, 0.16636206209659576, 0.22548669576644897, -0.0030091002117842436, -0.3977926969528198, -0.41521155834198, 1.7168575525283813, 0.12641535699367523, 0.18776969611644745, -1.0864441394805908, 0.0772535502910614, -0.5376574397087097, -1.291724443435669, 0.7249770760536194, 0.4712288975715637, 1.0704847574234009, 1.1162381172180176, -0.022483792155981064, 0.299452543258667, -0.7251756191253662, 1.129610300064087, 1.6787587404251099, -0.8878973722457886, -0.8346346020698547, -0.003994974307715893, -1.2489641904830933, 0.16433429718017578, -0.07838864624500275, -0.32839521765708923, 0.35549989342689514, -1.1300621032714844, -0.9560760855674744, -0.9136167764663696, -2.371781349182129, 0.9844303131103516, -0.6940805912017822, -0.24120311439037323, 0.4930407404899597, 0.7610227465629578, -0.5777279138565063, 2.298154354095459, 0.04704798758029938, -0.8708035945892334, 1.4734671115875244, -0.040750693529844284, -1.7444897890090942, -0.0808807834982872, -1.675829529762268, 1.4873933792114258, -0.14965705573558807, 0.1630428433418274, -0.2503108084201813, 1.2808656692504883, -1.40248703956604, -0.30422696471214294, -0.8425354361534119, -0.4825357496738434, -0.17554858326911926, -0.1495879888534546, -1.0794349908828735, -0.5677396655082703, -0.9968244433403015, 0.5743756890296936, -0.42487862706184387, -2.158679246902466, 0.815337061882019, 1.468827724456787, 0.9290169477462769, 1.7373595237731934, -0.8694317936897278, -0.3026522397994995, -0.43008482456207275, 1.4025260210037231, 0.8186401724815369, 2.878269672393799, -1.3851193189620972, -0.6090945601463318, -1.8337796926498413, 1.234424114227295, 0.3750101923942566, 1.225616693496704, 0.4248877167701721, -0.12866191565990448, -0.13567206263542175, -1.8210020065307617, 0.4881913959980011, 0.7110897898674011, 1.0648705959320068, 0.15623237192630768, 0.18588387966156006, 0.19460861384868622, -0.5831891298294067, -2.1153531074523926, 0.098716601729393, 0.016238996759057045, -0.37183982133865356, -0.12182419002056122, -0.09789914637804031, -1.3360214233398438, 0.022607970982789993, -0.4900563657283783, 0.8797481060028076, 1.0749963521957397, 0.1870768517255783, 0.4834185838699341, -0.1843733787536621, 1.8326425552368164, -1.4850609302520752, -0.5598868727684021, 0.4541190266609192, 0.527801513671875, -0.028080234304070473, 0.5105870366096497, 1.3661713600158691, 0.8246424794197083, 0.5770922899246216, -2.0362119674682617, -0.3539600372314453, 0.3605875074863434, -0.2247065156698227, -1.9311763048171997, -0.01048313733190298, -0.6987806558609009, -0.7197980284690857, 0.749445378780365, -1.057083010673523, 1.551312804222107, 1.7043170928955078, 0.24128805100917816, 0.17547866702079773, -1.4311769008636475, -0.5346030592918396, 0.7577619552612305, -0.21670277416706085, 0.20814792811870575, -0.5680732727050781, -0.7682902216911316, -4.0891804695129395, 0.5751938223838806, 0.20534446835517883, 0.5461596250534058, -0.5664836168289185, -0.09023275971412659, 0.9008268117904663, -0.7974669337272644, -0.17039278149604797, 1.2466480731964111, -0.45822110772132874, -0.09174130111932755, 0.08442575484514236, -0.7496356964111328, 0.029482705518603325, -0.8725888729095459, 0.27177736163139343, -0.2584070861339569, 0.2547352612018585, 0.36071938276290894, 1.9323066473007202, 0.26925066113471985, 0.9441603422164917, 0.37402084469795227, -0.1967223733663559, 2.8213446140289307, -0.34793582558631897, 1.6053603887557983, -0.6170011162757874, -0.8786789774894714, 0.09140071272850037, -0.8902190327644348, 0.059746891260147095, 0.41667941212654114, 0.5525617599487305, 0.6958832144737244, 2.083479642868042, 0.14716024696826935, -1.1668894290924072, 0.49732840061187744, 1.2149715423583984, 0.6047837138175964, -0.9489755034446716, 0.7458117008209229, -1.8078702688217163, 0.20024408400058746, 1.1448463201522827, -0.8690577149391174, -0.12487661838531494, -0.3586028516292572, 0.6213735938072205, -0.7996065020561218, -0.3384285569190979, -0.0751831904053688, 0.048335496336221695, 1.2217130661010742, 0.29063329100608826, -1.2351974248886108, 0.1853562295436859, -0.8981186747550964, -0.44564691185951233, 0.18652915954589844, -0.9668805599212646, -1.2395631074905396, -0.3152938783168793, -2.4219441413879395, -0.16704103350639343, 0.14912737905979156, 0.007349896244704723, -0.6220409870147705, 0.791490375995636, 0.27221232652664185, -0.24620993435382843, 0.7479798197746277, 0.7170540690422058, -0.3907899558544159, -0.1781703233718872, 0.3457030951976776, 0.9051235318183899, -1.530714988708496, 0.1372728943824768, 0.18012917041778564, -0.06451054662466049, -0.34572550654411316, 0.05197186395525932, -0.9687296152114868, 1.8844859600067139, -2.413867712020874, -0.7346579432487488, -1.4147509336471558, 1.0781428813934326, -0.2925129532814026, -0.6029458045959473, 1.706664800643921, 0.5102226138114929, 1.5846229791641235, 0.7744362354278564, 0.553753137588501, 0.2235374003648758, -0.41425621509552, 0.1458737701177597, 1.100627064704895, -0.25410139560699463, -1.663576602935791, 1.2044422626495361, 1.1124224662780762, -1.6942802667617798, -0.019833140075206757, -0.7205392718315125, -0.10228483378887177, 1.08090341091156, -0.2620464861392975, -1.793483853340149, -1.6179101467132568, 0.3186570405960083, 0.7023363709449768, -0.10412706434726715, -0.9221096038818359, -1.0913456678390503, -0.10334106534719467, 1.3460280895233154, 1.0858522653579712, -0.4690788686275482, -0.4088802933692932, -1.545951008796692, 0.9312272071838379, 2.0703916549682617, 0.19881580770015717, -0.07743921130895615, 1.5374174118041992, 0.013673888519406319, -0.8878146409988403, 0.8585830330848694, 0.1518324464559555, -0.7440785765647888, 0.3707291781902313, 2.192277669906616, 0.2988615036010742, -0.0274309404194355, -0.09268512576818466, -0.6076547503471375, -2.141274929046631, 0.9754651784896851, 0.9539973139762878, -0.22619588673114777, -0.6938020586967468, 1.1562459468841553, 0.009319578297436237, 3.2106211185455322, -1.3047425746917725, -0.4966906011104584, -1.0256160497665405, -1.8924176692962646, 1.1110687255859375, -1.0267715454101562, -0.5696208477020264, 0.0931713730096817, -0.35795632004737854, -0.15833738446235657, -1.0705807209014893, 0.31994760036468506, 1.9917546510696411, -1.4191707372665405, 2.0829286575317383, 0.2149810642004013, 0.13723939657211304, -1.4991047382354736, -1.4910194873809814, -2.142421007156372, -1.3177425861358643, 0.1578812450170517, -0.08790460973978043, -0.8589195013046265, -0.7549856305122375, 1.871612548828125, -0.19939129054546356, -0.6324578523635864, 0.18127982318401337, -0.7280901074409485, 0.03971448540687561, -1.6686456203460693, 1.1761188507080078, -0.5131842494010925, 1.721113920211792, -1.4790080785751343, -0.47363734245300293, 0.4598199129104614, 1.5179404020309448, 0.7895197868347168, -0.5017338395118713, 0.5811826586723328, 1.573438286781311, -1.7282921075820923, -1.3167915344238281, -1.7219918966293335, -0.6094489097595215, 0.4161106050014496, 0.5214106440544128, 0.9762741327285767, -0.30874037742614746, 1.0399905443191528, -0.6634269952774048, 0.9818903207778931, 0.46670886874198914, -0.29988279938697815, 0.7758249640464783, 0.1547810584306717, 1.1798728704452515, 0.3508491516113281, 0.6016925573348999, 0.4250056743621826, -0.35040178894996643, 0.24708521366119385, 0.4322541058063507, 1.4014798402786255, -0.9862702488899231, 1.4326223134994507, -1.18943452835083, -0.8596220016479492, -0.04097719117999077, -0.4871036410331726, -0.7398184537887573, -0.9241726994514465, -0.920226514339447, -1.169289469718933, -0.35362687706947327, -0.645318329334259, 1.8716970682144165, 1.150520920753479, -0.0330018512904644, 1.677506446838379, 1.0636203289031982, -0.8688413500785828, 1.218526005744934, 1.4019792079925537, -0.6767911911010742, -0.6470938324928284, -1.4328891038894653, -0.002338529797270894, 0.20676977932453156, -2.592256546020508, -0.48144176602363586, 0.9974931478500366, -1.581196665763855, -0.7994017004966736, -1.1868189573287964, -0.14380232989788055, -0.2709015905857086, 0.9405631422996521, 0.024370621889829636, -1.4045449495315552, 2.2812459468841553, -1.1807512044906616, -0.4403737485408783, -0.2598359286785126, 0.44376683235168457, 0.07204659283161163, -0.5378012657165527, 0.1415334939956665, 1.1339728832244873, -1.4211242198944092, 1.160253882408142, 0.19918105006217957, 0.6120874285697937, -0.5541191697120667, 0.7429214119911194, -0.015211338177323341, 0.24141816794872284, 1.5385465621948242, 0.16902020573616028, 1.7234004735946655, -1.0579783916473389, -0.41190820932388306, 0.40808728337287903, -0.6405631303787231, -1.3015635013580322, 0.93513023853302, -1.1526256799697876, 1.9167143106460571, -0.21650215983390808, -0.30982816219329834, -0.9371294379234314, -0.3077278137207031, -1.502986192703247, -0.28976210951805115, 0.15164296329021454, 1.1932579278945923, 0.004727821797132492, -0.5793572664260864, -0.09043490141630173, 0.5398175120353699, -0.8981172442436218, -0.32874390482902527, -0.8589177131652832, -0.2928140461444855, -0.1761009842157364, -0.8363795280456543, 1.824951410293579, 0.20487089455127716, -0.6164351105690002, -1.1102392673492432, 0.3120127022266388, -0.23955629765987396, -0.062392473220825195, 2.812934160232544, -1.5007048845291138, 1.615919589996338, 0.41766393184661865, -0.7747771143913269, 1.2475453615188599, -0.2996961176395416, 0.6269049048423767, 0.42790400981903076, -0.2254326492547989, 1.5713233947753906, -0.5210096836090088, 0.6404090523719788, 0.1765509694814682, 2.226083517074585, -0.5879830121994019, 0.7956436276435852, -1.1540013551712036, 1.0440406799316406, 0.470908522605896, 0.1787223219871521, 1.3723117113113403, -1.8089172840118408, 0.19240571558475494, -2.563847064971924, 0.4917314350605011, -0.445403516292572, 1.6911803483963013, 0.22340261936187744, -1.4686676263809204, -0.3821084797382355, 0.2350626289844513, -0.8054102659225464, -2.5755629539489746, -0.8529844880104065, 1.838570475578308, -0.7074824571609497, 0.40974506735801697, 2.531829595565796, 1.4217944145202637, -0.32872703671455383, 0.6549538373947144, -0.9341892004013062, 1.5117260217666626, 0.41296738386154175, 0.14195184409618378, -1.3473236560821533, -0.06976277381181717, 1.0096789598464966, -1.6161142587661743, -0.03766705468297005, -0.7468438744544983, 0.1549932211637497, 0.5360715389251709, -1.2591142654418945, 0.2505013048648834, 1.416849970817566, 0.5048378109931946, -1.0913598537445068, 0.5858504176139832, 0.1255234032869339, -0.1823478788137436, 0.8276010751724243, 2.071049690246582, -0.4919072985649109, -0.7337844371795654, 1.043181300163269, 1.4334908723831177, 1.0141111612319946, 1.4030420780181885, 1.2715030908584595, 0.24590179324150085, -0.35744383931159973, -0.2025143951177597, 0.07785602658987045, 0.24807123839855194, -2.227081537246704, 0.7964541912078857, 0.2197781503200531, -0.3013516068458557, 0.2381857931613922, -0.05312345176935196, 0.02633816748857498, 0.15970858931541443, -0.17623184621334076, 0.3133963346481323, 0.09542571753263474, -0.9898251891136169, -2.5636579990386963, 1.3486833572387695, -0.7682626843452454, -0.3789953291416168, -0.35180026292800903, -1.307154655456543, 1.1562073230743408, 2.054166793823242, 0.9888807535171509, 0.5246323943138123, -0.528226375579834, -0.3621760606765747, -0.9378591775894165, -0.01663210429251194, -0.7882599234580994, 0.5919718742370605, -0.07719402015209198, 1.1112092733383179, 1.230840802192688, 0.8393102288246155, -1.6907809972763062, -1.8801144361495972, 1.5332437753677368, -0.8635753989219666, 0.7704023122787476, 1.7292379140853882, 0.4030991196632385, -0.2050492912530899, 1.0591546297073364, 1.749776005744934, 0.8783243894577026, -0.6044779419898987, -0.5275940299034119, 0.8030275106430054, 0.006521960720419884, -0.7487226128578186, 0.9676187038421631, -1.333808183670044, -1.8523762226104736, 0.6477354168891907, 0.3251427710056305, -0.7181235551834106, 0.027796948328614235, -1.3965898752212524, 0.5676904320716858, -0.02659877948462963, -0.04100853577256203, 0.4557773768901825, -0.493967205286026, -0.5989763736724854, 0.708041787147522, 0.870983898639679, -0.859614908695221, 0.9232622981071472, 1.2786146402359009, -0.7372002005577087, 0.6971710324287415, -0.5304075479507446, -0.5317458510398865, 1.1948069334030151, -0.4252788722515106, -1.0849440097808838, 0.3861459493637085, 0.0023976422380656004, -0.1939876824617386, 1.2361297607421875, 1.1061208248138428, 1.002912998199463, 0.7871919870376587, -0.07821064442396164, -0.829177975654602, -1.407829999923706, -1.8177103996276855, -1.1032984256744385, -0.08212627470493317, 0.6475955247879028, 0.7435848712921143, -0.1801932454109192, -0.9504362344741821, -0.8276994228363037, 0.13445892930030823, -0.5955163836479187, -0.8830939531326294, -0.4478631913661957, -0.19090785086154938, -0.10406293720006943, 0.284018874168396, -0.030961083248257637, 0.2097092568874359, -1.2605359554290771, -1.0218011140823364, -0.3789474070072174, 0.47684285044670105, 0.23412634432315826, -0.8104138374328613, -2.831368923187256, 1.1320995092391968, -1.5464563369750977, -0.1802695095539093, 0.8293907046318054, -0.38508301973342896, 0.3618783950805664, -1.1389316320419312, -0.4747505486011505, -1.299362063407898, 1.0710511207580566, 0.6789649724960327, -1.1531002521514893, -0.14933530986309052, 0.7349866628646851, 0.18286170065402985, -0.06480859965085983, 1.3740992546081543, 0.43465039134025574, 0.5511179566383362, -0.2828735113143921, 1.0023863315582275, -0.23504342138767242, 0.4326840341091156, -0.5179474949836731, 1.4779058694839478, -0.6222383975982666, -0.6333101987838745, -0.34190213680267334, 0.27761170268058777, -0.03029446117579937, 1.3347891569137573, -0.6890266537666321, -0.2591013014316559, -0.3036549985408783, 0.11255618184804916, -0.9105563759803772, -1.3712269067764282, -0.8861827254295349, -1.113930583000183, -1.0083506107330322, 1.027623176574707, -0.479660302400589, 0.9444434642791748, -0.9111537337303162, -0.5343816876411438, -0.8317335247993469, -0.745829164981842, 0.2230522632598877, 0.6097347736358643, -0.4588974416255951, 1.2609704732894897, 1.3406498432159424, -0.4459143579006195, -0.017428722232580185, -0.3249419629573822, 0.9216673970222473, 0.7611348032951355, 0.260627418756485, 1.8624482154846191, 0.63743656873703, 0.4371151626110077, -0.04020722582936287, 0.7507811188697815, -1.0260121822357178, -0.8873087167739868, 1.3268344402313232, 0.2497665286064148, 1.16286039352417, 0.18332195281982422, 1.2323654890060425, -1.4850044250488281, -1.2300227880477905, 1.4642378091812134, 0.06600392609834671, 0.6982557773590088, -1.301161289215088, -0.5728763341903687, 0.8856787085533142, 1.8556690216064453, 0.05409965664148331, -1.1408193111419678, -0.009207471273839474, -1.5124627351760864, -1.1118557453155518, -0.9918838739395142, 0.35601747035980225, 0.680908203125, -1.597776174545288, 0.7251871824264526, -1.491514801979065, -0.4395568072795868, 1.3350436687469482, 1.223861575126648, -0.9580800533294678, 0.8902193307876587, -0.6540669798851013, -0.6658568978309631, 1.7211246490478516, -1.1560019254684448, 1.7954492568969727, -0.9847183227539062, -0.26052388548851013, -1.027720332145691, -0.613737165927887, 0.8725608587265015, 0.8705886602401733, -1.8578850030899048, -1.2868013381958008, -0.15462274849414825, 0.9168114066123962, 0.9895310401916504, -0.16072100400924683, -0.3187059462070465, -0.46581554412841797, 2.2952680587768555, 0.0898311510682106, 0.04836924374103546, 0.3250260353088379, 0.371491938829422, -0.2795222997665405, -0.31770867109298706, 0.937502384185791, 1.8814325332641602, 1.5265322923660278, 0.5030299425125122, -0.9911047220230103, 0.24304768443107605, 1.6065057516098022, 0.8362842798233032, 1.5945663452148438, -0.08823832124471664, 1.2013365030288696, 1.4742004871368408, -1.0367239713668823, -0.5117027163505554, 1.8280421495437622, -0.2998265027999878, 1.1453243494033813, -1.8083889484405518, -2.5562045574188232, 0.2169840782880783, -0.3875008225440979, 1.1155146360397339, 0.9418060779571533, 0.268808513879776, -2.928877353668213, -0.6853763461112976, -0.27321574091911316, 0.5326576828956604, -0.12063124030828476, -0.24373947083950043, 0.3075070083141327, -0.06940872967243195, -0.7883878946304321, -0.6324031352996826, 0.7992472052574158, 0.26906612515449524, 0.14192312955856323, -0.2028111070394516, -0.6878352165222168, -0.11958455294370651, 0.9213016033172607, -0.31517288088798523, -0.5596932768821716, -0.4496490955352783, 1.9487781524658203, -1.0551366806030273, 1.8369039297103882, -0.003998946864157915, -0.10686474293470383, 2.0660452842712402, -0.5217623114585876, 1.5129305124282837, 0.804219126701355, -0.6610397696495056, 0.3475760221481323, -1.1031367778778076, -0.3688061237335205, 0.9775188565254211, 1.1924526691436768, -0.8772134184837341, 1.4406405687332153, 0.0554826594889164, -0.5612605810165405, 0.9757028222084045, -0.4838300943374634, -0.39572587609291077, 0.0007713143713772297, -0.531912624835968, 0.44649267196655273, 1.330918312072754, -0.29254353046417236, -1.0933445692062378, 0.1488153636455536, 0.541679859161377, -0.9732730388641357, -0.4080232083797455, -0.5626826882362366, -0.12649263441562653, -0.8052788376808167, 0.6837653517723083, -1.3276710510253906, 0.5236856937408447, -0.8963623046875, -1.158416748046875, 0.41026270389556885, 1.0607304573059082, 0.22651104629039764, 0.5701780319213867, -1.5843961238861084, 1.0528903007507324, -1.448905348777771, -0.6449064612388611, 2.0617284774780273, -1.65912926197052, -1.3615992069244385, 0.5650781989097595, -0.2629377245903015, -0.30222755670547485, 0.42424628138542175, -0.003630791325122118, 0.0815185084939003, -0.9733566045761108, 0.16563770174980164, -1.0842981338500977, -1.1879515647888184, 1.832540512084961, -1.3412108421325684, 0.2287040650844574, 0.2430390864610672, 1.845773458480835, 0.7032935619354248, 0.9097131490707397, 1.1260350942611694, -1.261823296546936, 0.010894951410591602, 0.5724250674247742, -0.6231943368911743, 0.4919807016849518, -0.46040114760398865, 0.6863197088241577, -1.1149623394012451, 1.742306113243103, 0.5650222897529602, 1.211181879043579, 0.495632141828537, -0.5384053587913513, 0.23160913586616516, 0.934517502784729, 1.902078628540039, 0.09719064086675644, 0.8527063727378845, -1.0127227306365967, 0.3732776343822479, 0.40605905652046204, -0.6339533925056458, 1.0098652839660645, 0.7978814244270325, 0.00851534865796566, 1.2808117866516113, 1.5511661767959595, 0.8107873797416687, 1.1163560152053833, 1.4958916902542114, -0.7419031858444214, -1.5314692258834839, 0.8616582155227661, -0.9755992889404297, 1.2949267625808716, 1.8538568019866943, 0.9894458651542664, -1.8593933582305908, -0.020616564899683, -0.6185851693153381, -0.30882206559181213, 0.29528510570526123, 0.2987560033798218, 0.6297689080238342, -0.04295200854539871, -1.2449486255645752, 0.6358973979949951, -0.713229238986969, -1.555290937423706, -0.07134435325860977, 0.10201860219240189, -0.7401140332221985, 1.2598865032196045, -0.49359437823295593, 0.7823372483253479, -0.31475120782852173, -1.2692025899887085, -0.1203349158167839, -1.7697720527648926, 0.7380971908569336, -0.7811428308486938, 0.5202094316482544, 0.6640260219573975, -1.1612409353256226, -0.5471796989440918, -0.21266445517539978, -0.8103829026222229, 0.017484556883573532, 0.19778725504875183, -1.2190767526626587, -1.1113137006759644, 0.6692700386047363, -1.1490846872329712, 0.32329675555229187, 0.10503442585468292, -0.3237253725528717, 0.8782718777656555, 0.7386998534202576, -1.4150205850601196, 1.4581650495529175, -0.20852074027061462, 0.7353355884552002, -0.44482967257499695, -0.8708253502845764, -2.417001485824585, -0.7927777767181396, -1.0088697671890259, -0.9204990863800049, -1.0152531862258911, 1.0020709037780762, 1.5581616163253784, -1.2502578496932983, 0.9238774180412292, -1.1208535432815552, -0.5384106636047363, 0.9857307076454163, 0.11339713633060455, -0.03508475795388222, -0.16631940007209778, 0.25431180000305176, 0.6946102380752563, 0.6179450154304504, -0.6721134781837463, -1.5246741771697998, 0.943008303642273, 0.04046637564897537, 1.237906575202942, 1.4564449787139893, -0.42140060663223267, 0.006835350766777992, -0.7724081873893738, 1.337955117225647, 0.8032686114311218, -1.6599345207214355, 1.2904595136642456, 1.0307236909866333, 0.7526267170906067, 0.5387735962867737, 0.3725251853466034, 2.921103000640869, 1.1166095733642578, 0.4336961805820465, 0.18268409371376038, -0.32464295625686646, 0.11278391629457474, -0.6871371269226074, 0.32118380069732666, -0.6618940234184265, -0.4590485394001007, -0.4045164883136749, -0.6066463589668274, -0.5318851470947266, -2.2094905376434326, 1.2480614185333252, -0.5160583257675171, 1.6358169317245483, 0.3255199193954468, 0.24010737240314484, -0.5585188269615173, -1.528343915939331, 0.9444664716720581, -0.880872368812561, -0.19331806898117065, -0.3822624683380127, -1.346784234046936, -0.01598103530704975, -1.300421953201294, 0.1409977227449417, -0.1576232612133026, -0.37121912837028503, 1.256303071975708, -1.953783631324768, -0.4784015417098999, -2.2236876487731934, -0.0682695135474205, -0.0954732671380043, 1.1594516038894653, 1.2445858716964722, 1.2031941413879395, -1.510124683380127, -1.1735907793045044, 0.7561752796173096, 0.1246083527803421, 0.12041227519512177, 0.33651167154312134, 0.6066470146179199, -2.465076208114624, -0.4289446175098419, 0.29807528853416443, -0.055176619440317154, -0.5030565857887268, 0.39845654368400574, 1.1288429498672485, 0.8532119989395142, 1.081512451171875, -0.2298566997051239, -0.92501300573349, -0.4139283001422882, 0.1851506382226944, 2.8394367694854736, -1.1180623769760132, -0.28056198358535767, 0.8965258598327637, -0.9785715937614441, -0.29936376214027405, 1.2698521614074707, -0.6548411250114441, -0.08507397770881653, 2.3991496562957764, 1.6453661918640137, -1.8518977165222168, -1.4227303266525269, 0.39662158489227295, -0.775868833065033, -1.0180844068527222, 1.5034964084625244, 0.15989531576633453, -0.15448258817195892, 0.09268322587013245, 0.7998362183570862, -0.7486312389373779, -1.4421584606170654, 1.1618471145629883, -1.1404802799224854, -0.7748233675956726, 0.35164549946784973, -0.08562415093183517, -0.25798290967941284, -0.3413153886795044, -0.5018101334571838, -1.8124668598175049, -0.6100953817367554, -0.6954883933067322, -0.2108529955148697, -1.1776913404464722, -1.4073436260223389, -1.3347158432006836, -0.21578703820705414, 1.8953794240951538, -0.24378371238708496, -0.5615745782852173, -0.8333798050880432, 0.4139077365398407, -0.6854944229125977, -1.4388409852981567, 0.10999634116888046, -1.247051477432251, 0.5202534198760986, 0.17554457485675812, -0.547232449054718, -0.4806600511074066, -0.21777845919132233, -1.0092755556106567, -1.9206346273422241, -0.06533913314342499, -0.53318852186203, 0.4997590184211731, 0.6964108943939209, -0.1610931009054184, 1.4936796426773071, 2.3609366416931152, 0.7118014693260193, -1.243803858757019, 1.8847405910491943, -0.23547321557998657, 0.454000324010849, 0.7263171076774597, 0.1944829523563385, 1.3362364768981934, 3.1190972328186035, 1.4887632131576538, -0.9472194314002991, 0.6351670026779175, 1.6188061237335205, -0.47981229424476624, 0.7137259840965271, -1.0675207376480103, -1.5443891286849976, -1.508752703666687, 0.10156601667404175, -1.0895315408706665, -0.2516680657863617, -0.5104069709777832, -0.43216851353645325, -0.27317366003990173, -0.6046749353408813, 0.1519629955291748, -0.3087390065193176, -2.3411786556243896, -1.2610570192337036, 0.40297186374664307, -0.21149493753910065, 0.2264309823513031, 0.942825436592102, 0.6375673413276672, -2.0321133136749268, 0.8000226616859436, -1.1482921838760376, 0.041047047823667526, 0.47873765230178833, 1.2192628383636475, 0.13228684663772583, -1.2212129831314087, 0.6625641584396362, 1.473014235496521, -0.8561541438102722, 0.5947011709213257, 0.24238139390945435, 1.8274472951889038, 0.9623329639434814, -0.3499490022659302, 0.6670490503311157, -1.3347325325012207, -0.6101311445236206, -0.660401463508606, -0.2017342746257782, -0.5133444666862488, 1.4304580688476562, 1.428741455078125, 0.012686943635344505, 2.376410484313965, -1.4227484464645386, -0.8271867036819458, -0.3095666468143463, -0.07455047219991684, -1.6790062189102173, 0.9467801451683044, -0.1709212064743042, -1.219269037246704, 1.2305694818496704, -3.0761258602142334, 0.7893452048301697, -0.32564958930015564, 0.13970603048801422, 1.2226642370224, 0.4839114546775818, -1.199885606765747, -1.1131712198257446, 1.5793427228927612, 0.22644998133182526, -0.19693389534950256, 0.2781292200088501, 0.01962096616625786, -0.33478227257728577, -0.15242500603199005, -0.6003096699714661, -0.8570120334625244, -1.40040922164917, 1.0392355918884277, -1.124435544013977, 1.8573918342590332, -0.15442515909671783, 0.3834000825881958, -0.07836654037237167, 1.1520432233810425, 1.597455382347107, -0.47449928522109985, -0.9841845631599426, -0.15492667257785797, 0.18940423429012299, -0.9528376460075378, 0.9718742966651917, -1.8503090143203735, 1.3008782863616943, -0.3329451084136963, -0.0005311006680130959, 0.7785916924476624, -0.41473355889320374, -0.32528573274612427, -1.01876699924469, 1.487260341644287, -0.8239650726318359, -0.558582603931427, -0.966096818447113, 0.834861159324646, -0.5131300687789917, 1.709224820137024, 0.10904666781425476, 0.7802101373672485, -1.6156604290008545, 0.08680537343025208, -0.7183433771133423, 0.2786146104335785, 1.6139764785766602, -0.6611647009849548, 0.1200856938958168, -1.2992933988571167, -0.4000803232192993, 0.1144610270857811, 0.07001549005508423, 1.4378643035888672, 0.19112907350063324, 0.04591482877731323, 1.0801174640655518, 1.2313318252563477, -0.5507512092590332, 1.1080858707427979, -0.5773521661758423, 1.5178601741790771, -1.6541922092437744, -0.04714972525835037, -0.4259544014930725, 0.9666258096694946, 1.120438575744629, 0.17019933462142944, 0.9809909462928772, 0.7126374244689941, 0.46835532784461975, 0.11259590834379196, -0.6453101634979248, 0.4813280403614044, -0.02371186390519142, 0.799165666103363, 0.6859090924263, -1.1963303089141846, -0.8081027269363403, -1.0654284954071045, -0.15754128992557526, -0.5446589589118958, 0.45949235558509827, 0.6034409403800964, 0.802539050579071, -1.3531849384307861, 0.015925642102956772, 0.3279239535331726, -2.089470148086548, -0.5736532807350159, -0.010251171886920929, 2.097425937652588, 0.7233757376670837, -0.9404290318489075, 0.3596610426902771, -0.9446700215339661, -0.4590325653553009, -0.9134833812713623, -0.7284244894981384, 0.1481374353170395, 0.6755565404891968, -0.6347372531890869, 0.2882252335548401, -0.8197693824768066, 1.387655258178711, -0.3870299756526947, 0.072164386510849, 0.19878189265727997, 0.946887195110321, 0.198572039604187, -0.6357409954071045, 1.0673528909683228, 1.853955864906311, -1.0164023637771606, -0.8843926191329956, -0.3383176922798157, -1.3131835460662842, 0.7179622650146484, 0.014145844615995884, 0.9725791215896606, -1.175458550453186, -0.3636317849159241, -0.3086002767086029, -1.0566426515579224, -0.2898518741130829, 0.30110934376716614, 0.5947937369346619, -0.0335644893348217, 0.9134402871131897, -0.30794328451156616, 0.23682954907417297, 0.2677428722381592, -0.05052361637353897, -1.8768305778503418, 1.2279962301254272, -1.1125582456588745, -0.4012278914451599, -1.269633412361145, -0.9683339595794678, 0.41215813159942627, 0.4366421699523926, -0.8067513704299927, 0.7199941873550415, -0.4304836392402649, 0.7304769158363342, -1.2017563581466675, -0.4171921908855438, 1.5965157747268677, -0.039736565202474594, -0.20138564705848694, 0.6051761507987976, 1.0036731958389282, -0.22568047046661377, 1.3300819396972656, -0.8600044250488281, -0.37675678730010986, 1.3330249786376953, 0.379070907831192, 2.5588855743408203, 0.8839048147201538, -0.6819366216659546, -0.3286709487438202, 1.1601142883300781, 0.06126909703016281, -0.4181468188762665, -0.36616551876068115, 0.3578958213329315, 1.5994642972946167, 0.594323992729187, 0.4368307292461395, -0.8365756869316101, 0.6951070427894592, 0.26442813873291016, -0.43042710423469543, -0.1264520287513733, 0.8651685118675232, -1.1193180084228516, -0.48820748925209045, -0.3387635350227356, -0.4674351215362549, 1.00194251537323, -0.6400647759437561, 0.47796759009361267, 0.37083905935287476, 0.2920527756214142, -0.6095500588417053, 0.46181684732437134, -0.9606425166130066, 0.5293442010879517, 0.16596220433712006, 0.4889988899230957, 0.28795912861824036, 0.4949093759059906, -1.4841346740722656, -1.1123957633972168, 0.022611791267991066, -2.90258526802063, 0.0965908020734787, 0.9692156910896301, 2.4755568504333496, -0.591697096824646, 1.2196788787841797, -1.757757544517517, 0.18382364511489868, -0.6467345356941223, -0.08321956545114517, 1.7717313766479492, 1.3144207000732422, -1.0686156749725342, -1.3309144973754883, -2.7383415699005127, 0.4416589140892029, 1.1609219312667847, -0.41358697414398193, 0.8904708027839661, -0.3681536316871643, 0.340622216463089, -1.2203869819641113, 1.1486260890960693, -0.07429418712854385, 0.2645094394683838, 1.2572815418243408, 0.15981054306030273, -1.6971863508224487, -1.5551975965499878, 2.0093462467193604, 0.1100340411067009, -0.7126637697219849, -0.1675623655319214, -1.3926528692245483, 0.5533943772315979, -0.7293813228607178, -0.23346100747585297, 1.2881165742874146, 0.9534900784492493, 0.8128751516342163, -0.538779616355896, -1.3164823055267334, -1.6026623249053955, -1.426377296447754, 2.2895755767822266, 0.8745657801628113, 0.7427195310592651, 0.5073792338371277, -0.0731484666466713, -0.4115176200866699, 0.4118328094482422, -0.06717906892299652, 0.19791553914546967, -0.340227335691452, -0.8028042316436768, -0.5333346724510193, 0.2908779978752136, -0.3460550308227539, 0.6480700373649597, -0.9158581495285034, -1.1794787645339966, 1.0499804019927979, -1.3341811895370483, 0.08580602705478668, 0.6421428918838501, 1.4338382482528687, 1.131238341331482, -1.4816786050796509, -1.0977333784103394, -1.8554760217666626, -0.9737727642059326, 0.173165962100029, -0.27218401432037354, -1.0008620023727417, -2.1321239471435547, -0.3761070966720581, -0.06972380727529526, -1.0863959789276123, 0.6804254651069641, 1.6678147315979004, 0.22075575590133667, -0.17463445663452148, -0.057233285158872604, 1.0500417947769165, -0.252316951751709, 1.1997904777526855, -0.6048153042793274, 0.8240206241607666, 0.6886947154998779, 1.6000404357910156, 0.39460277557373047, 0.8994660377502441, 0.5039265751838684, 0.4933125376701355, 1.3542299270629883, -1.9486404657363892, -0.45290520787239075, -0.006811782717704773, 0.4803822636604309, 0.6807829737663269, -0.2724911570549011, -0.5218591690063477, -1.3441967964172363, 1.040645956993103, 1.2642152309417725, 1.7506499290466309, 1.3581695556640625, 0.6860989928245544, 2.0028018951416016, -0.5029283165931702, -0.07145271450281143, 0.5567953586578369, 1.113540768623352, -0.5920451283454895, 0.3567642867565155, -1.0933701992034912, -0.8279234766960144, -1.0811513662338257, -0.8024916648864746, 1.5425771474838257, 0.428303986787796, 2.3877456188201904, -0.4304000735282898, -0.7350414395332336, -0.5692071318626404, -2.017296075820923, -0.5336288213729858, -0.1323496252298355, 0.7280630469322205, 1.055224895477295, -0.38442903757095337, -1.0890673398971558, 0.8053601980209351, 0.11000275611877441, 0.6665964126586914, 0.3196125030517578, 0.7678747773170471, -1.4321303367614746, -0.07461020350456238, -0.0721748098731041, -1.4895424842834473, -0.4712409973144531, -1.760999321937561, 0.46802565455436707, 1.2006328105926514, -0.7833874225616455, -1.7969136238098145, 0.45080700516700745, -1.345665693283081, 0.6659373641014099, 2.295332908630371, 2.0461952686309814, 0.8991166353225708, -1.3355352878570557, 0.696819007396698, -0.2761078178882599, -0.640068531036377, 1.1384004354476929, 2.425143241882324, -0.5273287892341614, 1.6436481475830078, 2.3548367023468018, 1.6012128591537476, -1.4388104677200317, 0.8602041602134705, -0.49318382143974304, 0.23866841197013855, 0.24300320446491241, -1.5273337364196777, 0.3443683087825775, -0.9296454787254333, -0.344113826751709, 0.8569847345352173, -1.124709129333496, -0.0304167028516531, -0.6061316132545471, 1.9915474653244019, -0.9650951027870178, 1.2780461311340332, 0.0718873143196106, 0.6400179862976074, 1.293124794960022, 0.6701552271842957, -0.18756631016731262, 0.5618019700050354, 0.9299163222312927, 0.6263153553009033, -0.597235381603241, 0.4048302471637726, -1.0216960906982422, 0.6126073002815247, -0.7784113883972168, 0.3356756865978241, -0.9640005826950073, 0.4851296544075012, 0.29722052812576294, -0.09983745217323303, -0.3357236683368683, -0.5759318470954895, 1.0835338830947876, -1.228040337562561, -0.9762663841247559, -0.0716300904750824, -1.3669803142547607, -0.15412448346614838, 0.3915111720561981, -2.2258460521698, 0.4483397901058197, -0.2871856391429901, -1.1795474290847778, 0.27724745869636536, 0.4722493886947632, 0.9934136867523193, 0.07424064725637436, 1.9045833349227905, 2.032761335372925, -0.8451747298240662, 0.038420699536800385, 0.5888586044311523, -1.5726127624511719, -0.6464982628822327, 0.06541590392589569, -1.0965572595596313, -0.044520217925310135, 0.7814393043518066, 1.1026434898376465, 0.002721871715039015, 1.5250083208084106, 0.42872878909111023, 0.8226150274276733, -0.587145984172821, -1.4427088499069214, -0.7431302070617676, -1.1958483457565308, -2.4143433570861816, 0.8318676352500916, -0.6916522979736328, -2.0545654296875, -0.08236442506313324, -0.8214868307113647, -0.544745922088623, 1.0736253261566162, -1.5989490747451782, 0.3759484589099884, 1.5739268064498901, -0.778446614742279, -0.7421843409538269, -0.5285619497299194, -0.23325634002685547, 0.5700114965438843, -0.3475629687309265, -0.8233407139778137, -0.4333859980106354, -0.7344603538513184, -0.3002402186393738, -0.6298611760139465, -0.3233531415462494, 0.5885854959487915, 0.887384295463562, 1.402587652206421, -1.4176870584487915, 1.0724074840545654, 1.1804620027542114, 0.05452144518494606, 0.1418859213590622, 0.5480227470397949, -0.24875377118587494, 0.038179654628038406, 0.024801770225167274, -1.8836575746536255, 0.3853185772895813, 0.19371084868907928, -0.27925869822502136, -0.3157794773578644, -0.25377607345581055, -0.4943624436855316, -1.2470391988754272, -0.6909909844398499, -0.10169997811317444, -0.6903703808784485, -0.5525195002555847, 0.5361207127571106, -0.361164391040802, -0.7805944085121155, -0.6694285869598389, -1.0251893997192383, 2.475938558578491, 0.6902533769607544, -2.174668073654175, 0.19392529129981995, -0.4285900890827179, 1.499231219291687, -1.0602507591247559, -0.2195098102092743, -0.2567627727985382, 1.1121760606765747, -0.06387439370155334, 0.01723278872668743, 0.8520316481590271, -0.3634844720363617, -0.21786902844905853, -0.34451472759246826, 0.32840266823768616, 0.5833337306976318, 0.020736563950777054, 0.25339803099632263, 0.445429265499115, 0.4028330147266388, -1.345240592956543, -0.15899954736232758, -1.748542070388794, -0.30182090401649475, -0.14979799091815948, 0.3299565315246582, 2.0742452144622803, -1.0529768466949463, 1.9636462926864624, 1.1888796091079712, 1.2330952882766724, 0.010589645244181156, -0.7686272859573364, 1.5950570106506348, 1.9380353689193726, -1.2135040760040283, 1.0065640211105347, -1.5914080142974854, -0.3837207555770874, -0.5809981822967529, -0.9224341511726379, 0.7976959943771362, -1.596892237663269, 0.3342916667461395, -0.714226245880127, -2.0626842975616455, -0.245327427983284, 1.0711596012115479, -0.08295058459043503, -0.5486400723457336, -0.15699824690818787, 0.5662675499916077, -0.38474521040916443, -0.7794849276542664, -1.1153925657272339, -0.7481316924095154, -0.6679407954216003, 0.37003037333488464, 1.326358675956726, -0.08059101551771164, 1.7110074758529663, 0.7651911377906799, -0.3690924048423767, 0.3468093276023865, 0.7873550653457642, -1.3333858251571655, -0.9964739084243774, 0.11261626332998276, 1.2356528043746948, 0.6849446296691895, 0.5697582364082336, -0.9242404103279114, 1.1144293546676636, -0.355584979057312, -0.38005247712135315, -0.5978420972824097, -2.390651226043701, -1.1668989658355713, -0.37046992778778076, 0.19905968010425568, -1.098410964012146, 0.4388241767883301, -1.0934360027313232, 0.6862319707870483, 0.033731140196323395, 0.5834172368049622, -0.23697558045387268, -0.40488365292549133, -1.2639262676239014, 0.5398731827735901, 0.01101075392216444, -0.9796366691589355, -1.033857822418213, 1.3457144498825073, -2.2954511642456055, -0.44719383120536804, 0.6753028631210327, -0.4047189950942993, 0.9986732006072998, -1.9899085760116577, -0.4580492377281189, 0.630098283290863, -0.5303441286087036, -0.40574291348457336, 0.588839590549469, 0.29659005999565125, 0.3783121109008789, 1.4689561128616333, -0.0037292754277586937, -1.1574302911758423, 0.656889796257019, -0.04459914565086365, 1.515580177307129, 1.2594033479690552, -0.34462398290634155, 1.7635252475738525, -0.07170015573501587, -0.6277287602424622, -1.9554411172866821, -0.7723641991615295, 0.21073412895202637, 0.1547461748123169, 0.3402816653251648, 0.5617585778236389, -0.27729085087776184, 1.5949705839157104, 0.24217073619365692, -1.3996822834014893, -0.8924034237861633, 1.331654667854309, -0.8590887188911438, -0.08293239027261734, 1.0419659614562988, -1.4866200685501099, 0.20747606456279755, -0.11711055040359497, 0.954718291759491, -0.03313564881682396, -0.5877193808555603, 1.775842308998108, 0.29957008361816406, -0.5319972038269043, -0.07479316741228104, -0.15695732831954956, -1.3310835361480713, 1.7077680826187134, -0.17877532541751862, 0.327058345079422, -0.07687418162822723, 1.2303389310836792, -0.5555437803268433, -1.3894728422164917, 0.554940402507782, 0.8511759042739868, 1.5218713283538818, -0.05040712282061577, 1.0371558666229248, 1.0672805309295654, 0.3713819086551666, 0.4598217010498047, 0.3782026171684265, 0.8121109008789062, -0.14097900688648224, 0.9473346471786499, 0.3893072307109833, 0.6276816725730896, -0.3185805678367615, 0.22193589806556702, -0.30713874101638794, -0.4670068025588989, -1.284916639328003, 0.6359273791313171, -0.28194496035575867, 1.7963542938232422, -0.32459259033203125, -2.2661385536193848, -0.06083531305193901, -2.1714515686035156, 0.2527298629283905, -0.7318500280380249, -1.092583417892456, 2.3959879875183105, 1.2003562450408936, -1.90842866897583, -0.08977698534727097, 0.1833006888628006, 0.34098806977272034, 0.31240904331207275, -0.23253485560417175, -0.17405369877815247, 0.014285057783126831, -1.393506407737732, -0.8522031903266907, -0.49497756361961365, -0.627215564250946, 0.2287919670343399, -0.48014116287231445, 0.3729653060436249, 1.1137844324111938, -1.5729820728302002, 1.8027721643447876, -1.6250369548797607, 1.0314987897872925, -1.8917129039764404, 0.7754839658737183, 0.9790790677070618, -0.6437027454376221, -0.07703709602355957, -1.4749892950057983, -2.122065544128418, 1.0646549463272095, -0.4311472475528717, -0.8347219228744507, 1.1887325048446655, 0.2242131233215332, -1.8008369207382202, 0.24356842041015625, -0.19690454006195068, 0.3059843182563782, -0.6602296829223633, 1.7117114067077637, -0.5476590991020203, 0.12751084566116333, -0.27858734130859375, 0.7688169479370117, -0.06945139169692993, 0.8868463635444641, -0.14863598346710205, -1.0610541105270386, 0.6786877512931824, -0.7624992728233337, 0.4439639151096344, -0.6277350783348083, -0.03516438975930214, -1.10499906539917, -0.8338624835014343, 0.09722183644771576, 0.99824059009552, -1.6050220727920532, 0.7227344512939453, -0.5293151140213013, 2.107553482055664, 0.4289456307888031, -0.9550718069076538, -0.5112767815589905, 3.25549054145813, -0.5700991749763489, -0.1643134504556656, -0.20142923295497894, -1.142761468887329, 0.7743245959281921, 0.2822786569595337, -0.3097746670246124, -0.7749054431915283, -1.5035715103149414, 0.8545762300491333, 0.9525660276412964, -0.24149295687675476, 1.0045826435089111, -0.4331701397895813, 0.34673428535461426, 1.3677539825439453, 0.8124862313270569, -1.4397457838058472, 1.2783654928207397, -1.2479199171066284, 2.015137195587158, 1.4389406442642212, 1.0377740859985352, -0.3082456588745117, 0.09509171545505524, -0.9565531015396118, -1.551071047782898, -0.232014462351799, -1.1221615076065063, -0.18237078189849854, 1.3008472919464111, -1.0909918546676636, 0.08860337734222412, -0.30681276321411133, -0.38632357120513916, -1.2485837936401367, -1.0162882804870605, -1.520573616027832, 0.4125620424747467, -0.5120273232460022, 0.12409869581460953, -0.7657135128974915, -1.6145604848861694, 0.37512344121932983, -1.8976434469223022, 1.9551752805709839, 0.6118798851966858, -0.24550671875476837, 1.0432870388031006, 1.6341763734817505, -0.2414991557598114, 0.010005961172282696, 0.23869241774082184, 0.9901614189147949, 0.41994941234588623, 0.11125294864177704, 1.6693400144577026, 0.11310833692550659, 0.5960740447044373, 2.0983619689941406, 1.8450433015823364, 1.169641375541687, 1.543056845664978, 0.368586927652359, 1.2514586448669434, -0.4674871861934662, -0.6868473291397095, -0.21530520915985107, 0.7726234197616577, -0.2570609152317047, -1.0822594165802002, -0.42250606417655945, -0.11823126673698425, -0.20676708221435547, 0.8033221364021301, -2.7383694648742676, 0.35996121168136597, -0.6659278869628906, -0.9038200378417969, -0.18438054621219635, 0.1604645997285843, -0.4441521167755127, 1.3764479160308838, 1.241965651512146, -0.011208937503397465, -0.4876862168312073, -1.0065927505493164, 1.7737784385681152, 0.3957318663597107, 1.2911391258239746, 0.6980929970741272, 1.6554067134857178, 0.4439581036567688]], "dtype": "float32"}, {"__nd__": [1.1106700897216797, -0.15222714841365814, 0.4053272008895874, 0.40941348671913147, -1.7634084224700928, -0.2620294988155365, -0.6108601093292236, 0.3169506788253784, -0.5448534488677979, -0.4026315212249756, -1.14437997341156, 1.5275940895080566, 0.3476729393005371, -0.11144890636205673, 0.6177432537078857, 1.1533409357070923, -1.908816933631897, -1.4125481843948364, 0.666780948638916, 0.034891437739133835, 0.24616491794586182, -1.5756378173828125, 0.41136497259140015, -0.7704272270202637, 0.8075018525123596, 0.6497663855552673, 1.0004597902297974, -0.8059558272361755, -0.1297668069601059, -0.6156143546104431, 0.8716090321540833, 0.14067207276821136, 0.5824190378189087, -0.23139341175556183, -0.23413649201393127, -0.48614969849586487, 1.4800889492034912, -0.6590030789375305, -0.3675977885723114, 0.03827770799398422, -0.6936935782432556, -1.5058873891830444, -1.2192730903625488, 0.006794554181396961, 0.9475582242012024, -0.08699217438697815, 0.4471343755722046, 0.5975022912025452, -2.136979103088379, 0.48988035321235657, -0.9928496479988098, 1.6274055242538452, -0.0062637594528496265, 0.7175982594490051, -0.16529053449630737, -1.5900028944015503, -0.9796604514122009, -0.5902675986289978, 0.18471723794937134, -0.48596248030662537, 1.7606229782104492, 0.6928920745849609, 0.20124804973602295, 0.1015201285481453, 0.842535138130188, 0.44612735509872437, 0.7755951285362244, -0.17664839327335358, -0.9043435454368591, 1.3068159818649292, -0.7574070692062378, -1.863918423652649, -0.015478034503757954, -0.36503487825393677, 1.340457558631897, -1.4153186082839966, 0.004199593793600798, -1.8480331897735596, -0.884423017501831, -0.9792073965072632, -1.8122591972351074, 0.782747209072113, 1.2064566612243652, 0.9351854920387268, -1.5026875734329224, 1.640232801437378, 0.3647053837776184, 1.6113425493240356, 1.2969247102737427, -0.1806262880563736, 0.390463650226593, 0.6977787613868713, 0.6716805696487427, 0.6560057997703552, 1.9135100841522217, -0.3157995045185089, 0.1292300522327423, 0.8565001487731934, -0.13613639771938324, -0.5066201090812683, -0.007494912948459387, 1.8793389797210693, -0.31936293840408325, -0.43070846796035767, 0.09452731907367706, -0.28117963671684265, 0.9632620811462402, -0.3155350983142853, 1.8566985130310059, 1.0080288648605347, 1.2844537496566772, 1.0776937007904053, -2.7510364055633545, -1.7344328165054321, 0.13428673148155212, -1.3984777927398682, -1.032815933227539, 1.1633392572402954, -1.1260483264923096, -2.004777431488037, 1.5833386182785034, -0.34513625502586365, 0.19252903759479523, -0.7484891414642334, 0.5653030276298523, -0.6068191528320312, -0.2358635514974594, 0.3169682025909424, -1.3843042850494385, 1.6255475282669067, 0.09172826260328293, 0.08718269318342209, 0.002823509741574526, -0.24031582474708557, -0.7105770111083984, -0.7335501909255981, 1.0543484687805176, 0.21446624398231506, -0.009574473835527897, -0.5797610878944397, -0.20448584854602814, -0.08342427760362625, 0.6779265403747559, 0.4140389561653137, -0.6578171253204346, -0.1568610668182373, -2.5982179641723633, 0.061790380626916885, -0.04903554171323776, 0.06938077509403229, -0.10475665330886841, 0.9456052780151367, -0.17858146131038666, -0.5731289982795715, -1.160229206085205, 1.5433274507522583, 1.0809351205825806, 1.1533434391021729, -0.11154694855213165, -0.9009153842926025, 0.8240692615509033, 0.4035815894603729, 1.9478906393051147, 0.5145750641822815, 0.42698898911476135, -0.6025946736335754, -0.034637700766325, 0.18363186717033386, 1.076628565788269, 1.1708378791809082, 2.1984810829162598, 1.4487043619155884, -0.13367585837841034, 0.5729812979698181, 1.2285770177841187, -0.024524997919797897, 1.4503339529037476, 1.2516136169433594, 1.1107076406478882, 0.21550323069095612, 1.130369782447815, 0.47197943925857544, -1.085896611213684, -0.5040187835693359, 0.9117962121963501, 1.0749627351760864, 0.2589156925678253, -0.8944582939147949, 0.514035701751709, -0.9115947484970093, -3.0539145469665527, 0.6923102736473083, -1.1080106496810913, -1.7545571327209473, 0.8716346025466919, 1.584735631942749, -0.4075950086116791, -0.2500399947166443, -0.4609239399433136, -0.7559289932250977, -0.23064559698104858, 0.8422784209251404, -1.1789124011993408, 0.33003753423690796, -1.093101978302002, -0.2771548330783844, -0.9120185375213623, -0.08041658252477646, -0.4684227406978607, -0.22087149322032928, 0.6915638446807861, 0.0019718403927981853, -0.36940690875053406, -2.95172119140625, 0.8824265599250793, -0.10049163550138474, -0.5191078782081604, -0.3914644718170166, -0.016907602548599243, 2.147099494934082, 1.6568329334259033, 0.6910340189933777, 0.911100447177887, 0.9734358191490173, 0.08172591030597687, 1.503476619720459, 1.0943087339401245, 0.3828491270542145, -0.5947810411453247, 2.296887159347534, 0.884599506855011, -0.2483850121498108, -1.3085325956344604, 1.162505030632019, 1.1348628997802734, 1.1240835189819336, 1.4224823713302612, -1.637123465538025, 1.2042067050933838, 0.5988429188728333, -0.3687345087528229, -0.573197603225708, 0.7184597849845886, 1.4982596635818481, 0.9278473854064941, 1.2224594354629517, -1.074495792388916, -1.132305383682251, -0.19320650398731232, 0.1708480417728424, 0.3622971177101135, -0.4650070369243622, -0.5354860424995422, 1.0885543823242188, -0.10903019458055496, -0.7193660140037537, 0.2315131574869156, 0.643980085849762, -1.3279447555541992, -0.3203173875808716, -2.1234524250030518, -0.6180713176727295, 0.6223859786987305, 0.2743186354637146, -1.7345279455184937, 0.5019691586494446, 2.522071599960327, -0.8187070488929749, 1.3323743343353271, -1.7093640565872192, -0.5676312446594238, -1.1557751893997192, 0.12503547966480255, 0.7678846716880798, 0.2687581479549408, -0.7421426177024841, -0.7307009100914001, 0.5902741551399231, 1.2281227111816406, -0.5108302235603333, -1.3810368776321411, -0.11313477903604507, -0.11118653416633606, -1.248468279838562, -0.10608889162540436, -0.27042216062545776, -1.6072392463684082, 0.13822601735591888, 0.13150325417518616, 2.185201406478882, -1.32382071018219, -1.3163596391677856, -0.9268246293067932, -0.8835200071334839, -0.3618261218070984, 0.5322480797767639, -1.402494192123413, 0.08035250753164291, 1.0957043170928955, 1.159672498703003, 0.4486340284347534, -0.6241211295127869, -1.4725347757339478, 0.07813571393489838, 0.49415838718414307, 0.12040693312883377, 0.45569440722465515, 0.22529961168766022, -1.1784685850143433, -0.12352284789085388, 1.2400494813919067, 0.8494073748588562, -0.04285895824432373, -0.18935012817382812, 1.7305368185043335, 0.6454862952232361, -2.9556827545166016, 0.22552688419818878, 0.1564975529909134, 0.7825331091880798, 1.6678396463394165, 1.2425123453140259, 0.30550482869148254, -1.5855761766433716, -0.11245738714933395, -1.0578880310058594, -1.345382809638977, 0.7501911520957947, -0.7153807282447815, 1.0832072496414185, -0.08374189585447311, 0.4921894967556, -1.2023940086364746, 0.14648336172103882, 0.08459305018186569, -1.1569000482559204, -3.1079795360565186, -0.22215011715888977, -0.9649438261985779, 1.8107514381408691, 0.049979958683252335, 0.5748625993728638, 0.3563217520713806, -2.006582021713257, 1.781122088432312, -1.3999801874160767, 1.0879888534545898, -0.9983980655670166, 0.3355577886104584, 0.5598784685134888, -1.7903432846069336, 0.06943599134683609, 0.7655447721481323, -0.12379052489995956, 0.5877434015274048, -0.38202306628227234, 0.710078239440918, -1.1746790409088135, -0.17055316269397736, -0.7592929601669312, 0.37244588136672974, 0.9847657084465027, -1.155953288078308, -0.9565897583961487, -1.8565174341201782, -0.5742619633674622, -0.4733983278274536, 0.5178360939025879, 0.4340421259403229, 0.20533911883831024, 0.5969763994216919, 0.9624566435813904, 0.8270425796508789, 0.6932075023651123, 0.03555366024374962, -0.10422999411821365, -1.5079489946365356, 0.76307612657547, 0.2082880586385727, 0.7855660915374756, -1.430930495262146, 0.6219766736030579, 1.0457956790924072, 0.15681985020637512, -1.0871562957763672, 0.48465099930763245, 0.6218820810317993, -1.1213277578353882, -0.10440045595169067, -2.139824628829956, -0.21219615638256073, 0.6704493165016174, -1.0587345361709595, 0.6973420977592468, 1.155876874923706, -1.3578822612762451, -1.6474274396896362, -0.11189637333154678, -1.0720609426498413, -0.3939956724643707, 0.4487323462963104, -0.6666519045829773, 1.7976949214935303, 1.9211926460266113, 1.1551313400268555, 0.2648569643497467, -0.04630892351269722, -0.1450837105512619, -0.663616418838501, -0.1299779713153839, -0.7698073387145996, 0.19629742205142975, 0.7000443935394287, -2.177994966506958, 0.6180117130279541, -0.2906103730201721, 0.9335786700248718, -0.0663817897439003, -0.1683150678873062, -0.21380746364593506, 0.050431959331035614, 0.7577818036079407, 0.7214229702949524, -0.03284422308206558, 0.4578075408935547, -0.8698770403862, 0.7356573343276978, 0.8679177761077881, -0.5791230201721191, 0.36885613203048706, -1.0720717906951904, 1.8375765085220337, 0.8126088976860046, -0.7629533410072327, -0.3766633868217468, 1.2269059419631958, -0.8470953106880188, -0.2688260078430176, -0.6318812370300293, 1.075596570968628, 0.31988927721977234, -0.6047177314758301, -0.7609043717384338, 0.5236904621124268, 1.3702998161315918, 0.1634841114282608, -0.44886255264282227, 0.828194797039032, -1.185499906539917, -0.8419419527053833, -1.3992058038711548, -1.8074138164520264, -0.2632988691329956, -0.9450528025627136, -0.43249213695526123, -0.25454431772232056, 0.5233615636825562, 1.4586392641067505, -1.6241018772125244, 1.0326677560806274, -1.1236228942871094, 0.10852346569299698, 0.9516264796257019, -0.8059235215187073, 0.09193919599056244, 0.6036076545715332, 0.8503583669662476, 0.5157268047332764, -0.2305082082748413, 1.0518327951431274, 0.9454036951065063, 1.997957468032837, 0.5895649194717407, -0.430186003446579, -0.5020396709442139, -0.8344728946685791, -1.542920470237732, 1.2406872510910034, 0.5915588736534119, 0.2602403163909912, -0.9942725896835327, -1.1421571969985962, -1.0478570461273193, 0.19842347502708435, 0.4480571746826172, -1.5408432483673096, 0.16806718707084656, 0.9825604557991028, 0.4077802002429962, -0.19508376717567444, -0.9938583970069885, -0.667595624923706, 0.61257404088974, 3.4152321815490723, -1.0081497430801392, -0.11223208159208298, -1.5143483877182007, 0.19956684112548828, 0.44189977645874023, 0.5676705837249756, -0.3153025209903717, -0.5607629418373108, 1.777697205543518, -0.21385307610034943, 1.124035358428955, 1.1875274181365967, 0.4373512864112854, -0.8423541784286499, -0.6065123677253723, -0.7742213010787964, -0.3461708128452301, -0.5840067863464355, -0.08131670206785202, 0.5037803053855896, -0.19183139503002167, -0.16793952882289886, -0.6808256506919861, 1.455790400505066, 0.564695417881012, -2.226811647415161, 1.0623574256896973, 0.500760555267334, -0.9511191248893738, 1.6464229822158813, 0.1565520167350769, -1.2782596349716187, 0.5394327640533447, -1.8632200956344604, 1.940779447555542, 0.3255043029785156, 0.6027128100395203, 0.9628342986106873, -0.7064679265022278, -0.6573355197906494, 0.49485769867897034, -1.1247954368591309, -1.7418498992919922, 0.6769434809684753, 0.04088355600833893, 0.2709641456604004, 0.10602841526269913, -2.1107988357543945, -1.4879931211471558, -0.9782726168632507, 0.35216307640075684, 0.21356958150863647, -0.4688594341278076, -0.8433845639228821, -0.6740840077400208, 0.14466090500354767, 1.6294549703598022, -0.24672643840312958, 0.8291373252868652, -0.6807980537414551, 2.081165075302124, 0.5633008480072021, -1.8524945974349976, -0.261488139629364, 0.09947515279054642, 0.1543901562690735, 1.7521600723266602, 0.006100579630583525, 0.21010282635688782, 1.4230079650878906, 1.0427266359329224, 0.46416646242141724, -1.4267021417617798, -0.29561707377433777, -0.20729902386665344, 0.32124263048171997, -2.2940452098846436, 0.2201717346906662, 0.9035326242446899, 0.13542309403419495, 1.0108592510223389, -1.4293485879898071, 0.10394059121608734, 1.5279488563537598, 1.6206072568893433, -0.8912718892097473, -0.17259562015533447, 2.4473156929016113, -0.6225824952125549, -0.919701874256134, -1.3064627647399902, 0.5425514578819275, -0.13023026287555695, 0.16716283559799194, -0.7657462358474731, 0.5777683854103088, -0.7220314741134644, 0.5786172747612, 0.7118908762931824, 0.783793568611145, 1.461165428161621, -0.44937238097190857, 0.8036774396896362, -0.5577253103256226, -1.1882338523864746, -0.48436078429222107, 0.5873655676841736, 1.356074333190918, -0.0890907496213913, 0.3820250928401947, -0.42018482089042664, -1.3187156915664673, -0.6672170162200928, 0.7524144053459167, 0.17486333847045898, -0.390569269657135, 0.3065029978752136, -1.0032861232757568, 0.5639058947563171, -0.31473636627197266, -0.6954267024993896, 2.7497761249542236, -0.5987778306007385, 0.9945054650306702, 0.9804664254188538, 1.0528414249420166, -0.15742172300815582, -3.1562490463256836, -0.0005643739714287221, 2.522672176361084, -0.8294932246208191, 0.7664603590965271, 0.3323056995868683, -1.4010064601898193, -1.5718226432800293, 1.0654690265655518, -1.9385030269622803, -0.45766347646713257, -0.8474023342132568, -0.6010563373565674, -1.8013991117477417, 0.40974101424217224, -1.378421664237976, 1.775948405265808, 0.016240715980529785, 0.3305616080760956, 0.9939205050468445, -0.4386855363845825, 1.8491765260696411, -0.5167729258537292, 1.8790106773376465, 0.32081007957458496, 0.0651107206940651, -0.5341207385063171, -1.2667720317840576, -0.00993993878364563, 0.8980045318603516, 0.5278230309486389, 0.2010069042444229, -0.47867846488952637, -0.5116233229637146, 0.9839643239974976, 2.5144433975219727, 0.024528486654162407, -0.38721388578414917, -0.00044482474913820624, -1.2833656072616577, -0.0478840172290802, -0.6772815585136414, 0.20510978996753693, 0.46508002281188965, -0.8498406410217285, -2.0146596431732178, 0.4681808650493622, -1.30117666721344, 1.1955337524414062, 0.02730291523039341, -0.4859679341316223, -0.47617077827453613, 1.0762462615966797, 1.283730149269104, 0.26322081685066223, -0.5272166132926941, -0.5396133065223694, -0.9861959218978882, -0.6719396710395813, -0.37730270624160767, 0.22926653921604156, 1.075676679611206, -0.8880519270896912, -0.3182772099971771, 0.3931428790092468, -0.9253491759300232, -0.009433411061763763, 1.3312877416610718, 0.5400758981704712, -0.030030924826860428, 0.12553586065769196, 1.073796272277832, 1.1223905086517334, 0.11422967165708542, 1.4789613485336304, 0.6195327043533325, 0.9484044909477234, 0.09330173581838608, -0.21547174453735352, 0.1259307563304901, -2.1042673587799072, -1.3322266340255737, 1.8994286060333252, -0.43891292810440063, 0.8303403258323669, -1.7573403120040894, 1.7190178632736206, -1.1022026538848877, 0.09452790021896362, -0.46814724802970886, 0.7414917349815369, -1.6201298236846924, -0.0002631993847899139, 1.036165714263916, 0.13997173309326172, -1.268256425857544, -0.21248334646224976, 0.49463924765586853, -0.4700143039226532, 1.0894558429718018, -0.26071080565452576, 0.40042969584465027, 3.26560378074646, -1.424452304840088, 0.8819800615310669, -0.6695548295974731, 1.0770014524459839, 0.13416753709316254, -0.32077163457870483, 2.674943685531616, 1.1628304719924927, 0.09659633040428162, 0.2262973040342331, -0.13175958395004272, 0.11844263970851898, 0.40972501039505005, -0.9400756359100342, -0.72572261095047, -0.19537651538848877, 0.5415444374084473, -0.051075104624032974, 0.4336143434047699, 0.6809018850326538, -0.05128259211778641, -0.19335131347179413, -0.5493733286857605, -1.1483736038208008, 0.031886979937553406, -0.9328889846801758, -1.7143179178237915, -0.3230573832988739, -0.25877848267555237, 0.35702863335609436, -0.2718590199947357, -0.619476854801178, -0.7853025794029236, 1.3058714866638184, -0.3255458474159241, -0.5380864143371582, -0.7434488534927368, 1.3151932954788208, -0.806678295135498, 0.3036056458950043, -0.8051863312721252, -0.851664662361145, 0.5663772821426392, -0.02225782908499241, -0.42421025037765503, 0.5069081783294678, -0.5743657350540161, 2.0687150955200195, 2.270409107208252, -0.12005355954170227, -1.1554063558578491, -0.3949306309223175, -0.6393427848815918, -1.4283748865127563, 0.900152862071991, -0.1391717940568924, 0.5612910985946655, -0.6755608320236206, 0.2882300317287445, -0.6495807766914368, 0.5131465196609497, -0.5957773327827454, 0.1953066736459732, 1.0930044651031494, -1.2352927923202515, 0.3760751485824585, -0.5618935823440552, 1.8592771291732788, 0.17730207741260529, 0.004927332513034344, -0.5940876007080078, 0.33584874868392944, -1.1244933605194092, 0.01912521943449974, 0.006495264358818531, 1.4346710443496704, -0.3324205279350281, -0.10368824005126953, 0.20912569761276245, -0.7064511179924011, 0.18883351981639862, -0.5342404842376709, -1.0841556787490845, 0.30543044209480286, -1.373605728149414, -1.5564569234848022, 1.4153282642364502, 0.168373242020607, -0.8184921741485596, 0.9220093488693237, -0.026793811470270157, 1.0771634578704834, -0.3651280701160431, -0.0019634494092315435, -0.009624049998819828, 0.807813286781311, -0.3477253019809723, 0.6483062505722046, 0.5678465366363525, 0.9707841873168945, -0.8282725214958191, -0.10279464721679688, 0.9246404767036438, 0.08680717647075653, -1.14365553855896, -1.02934992313385, 0.021097583696246147, 0.4049299657344818, -0.6954467296600342, -0.8849969506263733, 0.6111257672309875, 0.06353044509887695, -1.2970809936523438, 0.2671439051628113, 0.1675429493188858, -1.2283496856689453, -0.2632337510585785, 1.2488105297088623, -0.0763634443283081, -3.025982141494751, -0.21862362325191498, 1.3512192964553833, 0.5248462557792664, -1.0522030591964722, -0.6470629572868347, 1.4559192657470703, -0.13144546747207642, 0.8255841732025146, -0.8019518852233887, 0.8177142143249512, -0.33830440044403076, 0.8335707187652588, -0.831494927406311, 0.6761385202407837, -0.524246096611023, -0.10490793734788895, -0.41106417775154114, 0.8186511993408203, -0.17134737968444824, -1.1593561172485352, -1.7918590307235718, 0.008715256117284298, 2.0650832653045654, 0.7753958106040955, 0.5755079984664917, 1.0371313095092773, 1.1288608312606812, -1.1997385025024414, -0.4068242907524109, 2.0727312564849854, 1.9405360221862793, -0.13458412885665894, -0.8844702839851379, 1.0459604263305664, -0.8120101094245911, -1.3277853727340698, -0.7399625778198242, -0.8886870741844177, 0.7209149599075317, -1.5613187551498413, 1.0340056419372559, -0.3476332426071167, -0.39318761229515076, 1.167709469795227, -1.203696370124817, -0.542036771774292, -1.6833704710006714, -0.030557088553905487, 0.4229543209075928, 0.5470391511917114, 1.1969941854476929, -1.0835031270980835, -1.1348944902420044, -0.3648506999015808, 0.3716816306114197, 1.1529268026351929, 0.5601277351379395, -0.7465309500694275, -0.12487179785966873, -0.3589063286781311, 0.5340082049369812, 0.05610279366374016, 0.53897625207901, -0.09721019119024277, -1.3656935691833496, -0.5220099091529846, -1.043124794960022, 0.2513294816017151, 0.9520591497421265, -0.28960710763931274, 0.6353975534439087, 0.18659870326519012, 0.17555399239063263, 0.5863285660743713, -0.26757749915122986, 2.020085096359253, 0.21522994339466095, -1.060664415359497, -0.6354893445968628, -0.16441786289215088, 0.34146687388420105, 1.5424435138702393, 0.381088525056839, -0.6939613819122314, -0.8463210463523865, -0.5396682620048523, -1.1142055988311768, 0.34493890404701233, -1.4185631275177002, -0.8256856203079224, 0.9836707711219788, -0.8446740508079529, -0.5066527724266052, 1.3015259504318237, 0.412867933511734, -0.05470651015639305, 0.8272185325622559, 1.0868233442306519, -1.4541339874267578, -0.5102556943893433, -1.8407528400421143, 2.9219281673431396, 2.0652809143066406, 1.2043062448501587, -0.929604709148407, -1.0857869386672974, -0.9196692109107971, 1.043652057647705, 1.551180124282837, 0.780029833316803, -0.3726823627948761, -0.9414969682693481, 0.5756073594093323, -0.3175600469112396, 0.6274101734161377, -0.37748822569847107, -0.2745189964771271, 0.8511030673980713, -0.9985618591308594, 1.3037506341934204, -0.6901049017906189, -0.1789197027683258, 0.3072677552700043, 0.8090299367904663, 0.2518833875656128, -0.8084534406661987, -0.00864393450319767, -0.44204258918762207, -0.2610566318035126, -0.9554079174995422, -0.42359188199043274, -0.14163708686828613, 2.1620075702667236, 0.01896400935947895, 0.2406599223613739, -2.7723135948181152, -1.014204740524292, -0.3722646236419678, 1.9971976280212402, -0.7337267994880676, 1.1235324144363403, -0.5246501564979553, -0.6074325442314148, 1.4515140056610107, 0.40992283821105957, -0.5666227340698242, -0.7529461979866028, 1.7942334413528442, 1.4110321998596191, -0.7194530367851257, -1.0413187742233276, 0.7014013528823853, -0.30862748622894287, 1.6426475048065186, 0.45713815093040466, -0.18634706735610962, 1.0505154132843018, -0.05997016653418541, 2.1529858112335205, 0.15556854009628296, 1.2108782529830933, 0.09300713986158371, -0.4069470763206482, 0.8961337804794312, 0.2796393930912018, 0.6164752244949341, 0.09869696944952011, -1.0279948711395264, -0.49536511301994324, -1.7154231071472168, 0.9867844581604004, -0.33297860622406006, 0.39618605375289917, -1.2339669466018677, 0.7561030983924866, -0.7510735392570496, 0.2523943781852722, -0.434390127658844, -1.1611319780349731, -1.2819368839263916, -0.7593214511871338, -1.025345802307129, -0.30479949712753296, 1.4448285102844238, -0.5747207999229431, -1.002082109451294, 1.626981258392334, 0.8107247352600098, 0.3816988170146942, -0.15379750728607178, 1.9448482990264893, -0.5759896039962769, -0.4682837426662445, 0.8457553386688232, 0.7090460062026978, 0.3169232904911041, 0.8844741582870483, -1.1552168130874634, 0.4290110766887665, -0.44605082273483276, -0.5507644414901733, -0.2732458710670471, -0.47346803545951843, 0.5139127373695374, 1.310582160949707, -0.624955415725708, 0.6318014860153198, 0.3496253788471222, 0.5941331386566162, 0.17388753592967987, -0.8313626646995544, -0.3926188349723816, -0.3735603392124176, -0.35229256749153137, 1.2371712923049927, 0.48315197229385376, -1.0925564765930176, -1.1347543001174927, 0.575421154499054, 0.09644723683595657, 0.28338295221328735, 0.8375856876373291, 1.480373501777649, 0.18855677545070648, 0.9558485150337219, 0.22227874398231506, 0.8783435821533203, -0.20069582760334015, 1.432051181793213, -0.6776766777038574, -2.539634943008423, -0.28070512413978577, -3.5307788848876953, 1.271059513092041, -0.086538165807724, -0.17599497735500336, 0.5001742243766785, 1.2370444536209106, -1.639357089996338, 0.028645463287830353, 0.3445426821708679, -0.26705771684646606, -0.47463253140449524, -1.1610056161880493, -1.0214587450027466, -0.062455400824546814, -1.8782294988632202, 1.0572580099105835, 0.06635146588087082, -0.6234825253486633, -0.21572881937026978, -0.3493844270706177, -0.8637118935585022, -0.7788625359535217, 1.3633114099502563, -0.47858917713165283, 0.6740795969963074, 1.962490200996399, -0.06542500108480453, -0.6343821883201599, 0.05654774233698845, -0.8433762788772583, -0.31798821687698364, -2.42858624458313, -1.3457531929016113, -0.5791599750518799, -0.5258304476737976, 0.12184575945138931, -0.026760585606098175, -0.6001641154289246, 0.8091926574707031, -1.163874626159668, 0.2205294370651245, 0.6383939981460571, -0.8499660491943359, 0.35316696763038635, -0.6473609805107117, -1.3719531297683716, 2.8955533504486084, 0.7763249278068542, 0.49949485063552856, 0.25475457310676575, 0.5391218066215515, -0.6473962664604187, -0.07684554159641266, -0.04241866245865822, -1.2493021488189697, -1.0737215280532837, 0.309497594833374, 0.24444693326950073, -0.7966497540473938, 0.7090653777122498, 2.5143280029296875, -1.0456751585006714, -0.8031325936317444, 1.251091718673706, -1.2600001096725464, -0.1729893684387207, 1.3523122072219849, -0.8479667901992798, 0.5611949563026428, -0.8503373861312866, -0.6071392893791199, 1.5043330192565918, -0.6136376261711121, 0.3258883059024811, 1.0582363605499268, -1.6298365592956543, -1.221681833267212, 1.1396403312683105, -0.3090377449989319, 0.1904553323984146, 0.3718738555908203, -0.19418418407440186, 0.07957437634468079, -0.03867310658097267, -0.4558497369289398, -0.14195039868354797, 0.7283473014831543, 0.11369173973798752, 1.6813833713531494, 1.0365726947784424, 1.1023751497268677, 1.9760410785675049, 0.024091007187962532, -0.9366785883903503, 1.293622612953186, 1.0533323287963867, -0.7002204060554504, 0.7313280701637268, 0.20562133193016052, -0.6639753580093384, 0.49345993995666504, -2.1757407188415527, 0.6846611499786377, -0.017301999032497406, 0.5280148983001709, -1.1088690757751465, -1.2004934549331665, 0.8610901832580566, -0.8986270427703857, -2.029834747314453, -0.38613125681877136, -0.9246924519538879, -0.9245842099189758, 0.205051988363266, 0.529403030872345, 1.6039326190948486, 0.7080954313278198, -0.4244546890258789, 1.2264955043792725, 1.4194104671478271, 0.1862957626581192, -0.5027457475662231, 1.1047250032424927, 1.3771812915802002, 1.3396246433258057, 0.34495726227760315, 0.10181194543838501, -0.9555750489234924, -2.1593122482299805, 0.5854294896125793, 0.3928143084049225, 0.3239555060863495, -0.48620468378067017, 0.603103756904602, 0.1820293366909027, 0.5053868889808655, 0.28904959559440613, -0.877373218536377, 0.22970564663410187, -0.7455713152885437, -0.20402690768241882, -1.2328847646713257, -2.1098172664642334, 0.7150433659553528, 0.44952598214149475, -0.0161418654024601, -0.3385765254497528, -0.16467344760894775, 0.41851532459259033, 0.6796219348907471, 0.6676524877548218, 0.917048454284668, 1.7608678340911865, 1.2221097946166992, 0.13994048535823822, 0.26941272616386414, 0.21477146446704865, -1.4766441583633423, -0.24285657703876495, -0.10494454205036163, 0.6047042012214661, 0.7195916175842285, -0.7336363196372986, 1.5132850408554077, -0.12433168292045593, 0.8596964478492737, -1.0654115676879883, 0.6945420503616333, -0.5427833199501038, -0.7005239129066467, -0.5574272871017456, 1.0798494815826416, 1.0008777379989624, 1.5260190963745117, -0.8716468214988708, -1.1288808584213257, 0.5414437651634216, -0.7850240468978882, -2.5598506927490234, 0.7420956492424011, -0.6231480240821838, -0.7864242792129517, 0.11116089671850204, 0.4805428981781006, -0.536954402923584, -0.9450905919075012, -0.5757330656051636, -0.179372176527977, 0.06522323191165924, 0.5360413789749146, -1.2007092237472534, -0.4074445366859436, 0.8840033411979675, 0.05712789297103882, 1.0222059488296509, 0.6987127661705017, -0.7124298810958862, -0.17178860306739807, 1.565048336982727, 1.6144025325775146, -0.9132906794548035, 1.880018949508667, 0.9375904202461243, -1.3444807529449463, -0.935899555683136, 0.8660380244255066, 1.3546502590179443, -0.05564349517226219, 0.5262272357940674, -1.1035358905792236, 0.1678779274225235, -0.49175989627838135, -1.1728094816207886, 0.8218696713447571, 0.007604923564940691, 0.860558032989502, 0.5829540491104126, -0.4195038378238678, 0.5356566309928894, -0.6702831387519836, 0.5812076926231384, -2.2641308307647705, 0.8786042332649231, -0.05936940759420395, 0.5888402462005615, 0.6006456613540649, 0.0822521448135376, -0.06696320325136185, -1.2943661212921143, 0.8209553360939026, -0.7996335029602051, 0.1144702211022377, 0.16248810291290283, -1.1631951332092285, 1.2213715314865112, 0.4156145751476288, -0.12432495504617691, 0.7612743973731995, -1.1996666193008423, 0.8452006578445435, 0.6415724754333496, -0.08671428263187408, -1.2364922761917114, -0.6370291709899902, 0.09265688806772232, -0.394604355096817, 0.7857913374900818, -0.7233502268791199, -0.7450869679450989, -0.1524098664522171, 0.3132070004940033, -1.048901081085205, 1.0134621858596802, 0.6861680150032043, -0.8376351594924927, 0.881447970867157, 1.8946115970611572, -0.45603248476982117, -0.21589067578315735, -0.6260625720024109, -0.8984400033950806, -0.9954155683517456, -0.8792785406112671, -1.3167498111724854, -0.8593196272850037, -0.10670086741447449, 0.030594216659665108, 0.4190877676010132, 0.03244265541434288, -0.158649742603302, -1.6249014139175415, -0.8407976031303406, 0.4723869562149048, -1.6342707872390747, -1.2476983070373535, 0.6719889044761658, 1.6262164115905762, -2.0218851566314697, 0.06443512439727783, -0.7103093266487122, -2.120495557785034, 1.875083565711975, 0.08801009505987167, -0.9744044542312622, 1.9726433753967285, 0.6030023097991943, -0.1089104413986206, -0.14930027723312378, -2.2370388507843018, -0.09818986058235168, -0.25724297761917114, 0.6626685857772827, -0.6295065879821777, 0.5817095041275024, -1.7918888330459595, 1.1165319681167603, -2.1329753398895264, 0.9923386573791504, -1.2854108810424805, -0.016841774806380272, 1.0344011783599854, -1.34834885597229, 1.8695669174194336, -0.11026006191968918, -0.30996668338775635, -0.8086133003234863, -0.2101781964302063, -0.4454749822616577, 1.49271559715271, -0.7549476623535156, -0.9529588222503662, 1.4723095893859863, -0.5805806517601013, -0.9506414532661438, -0.9662908315658569, 1.7081385850906372, 0.49461695551872253, 0.3429968059062958, -0.6757879257202148, 0.08933120965957642, 1.780019760131836, 1.3986506462097168, 0.14289619028568268, 0.4556300640106201, 0.15022172033786774, -0.1795087307691574, -0.255789577960968, 0.6891639232635498, 0.3663274943828583, -1.1893341541290283, -0.6546584963798523, 0.5703336000442505, 0.8572596311569214, 0.9181789755821228, -0.26913660764694214, -0.010514376685023308, -0.657398521900177, -1.4972418546676636, -0.4763447344303131, -0.37185245752334595, 0.4234803318977356, 1.6751642227172852, 1.0960936546325684, -0.131089448928833, -0.787172794342041, -0.35780346393585205, 0.9701912999153137, -0.2971879541873932, 0.4290289878845215, 1.5324901342391968, 0.6067636013031006, 1.2233976125717163, -0.6718936562538147, -1.3586007356643677, -1.0082142353057861, 2.2625672817230225, -1.021086573600769, 1.9930752515792847, 0.633464515209198, -0.01383341196924448, -0.7008122801780701, 0.8065764307975769, 0.31635719537734985, 0.6110134124755859, -0.5197698473930359, -0.3634992837905884, 0.3698939383029938, 2.0414748191833496, -0.47477957606315613, -0.26041990518569946, 0.5715094208717346, -0.2846997082233429, 0.8537763357162476, -0.9226614236831665, 1.4739952087402344, -0.19342270493507385, 0.6325192451477051, 0.7693546414375305, -0.41343674063682556, -1.9148683547973633, 0.28355735540390015, 0.8708300590515137, 0.11624881625175476, -1.0291434526443481, -1.2392163276672363, 0.840729296207428, -1.0256537199020386, 0.6503959894180298, 0.03956193849444389, -0.6272652745246887, -1.1560388803482056, -0.48144131898880005, 0.8073123097419739, 0.5906999707221985, -1.5966523885726929, -0.6401285529136658, -1.8829879760742188, -1.4807881116867065, 0.2701421082019806, 0.0061333379708230495, 0.6441782712936401, 3.2070772647857666, -0.24859771132469177, 0.5248255133628845, -0.332899808883667, 0.23963375389575958, -0.5441872477531433, 0.7492564916610718, -1.7652174234390259, 0.6093224883079529, -1.8045088052749634, -0.608969509601593, -0.33133867383003235, -0.24032807350158691, 0.29157230257987976, -0.9279182553291321, 1.1673446893692017, -0.6134234666824341, -0.5342817902565002, 1.6814615726470947, 0.9883540272712708, -0.47342225909233093, 0.5945858955383301, -1.0323067903518677, 1.1582618951797485, 0.25862088799476624, -2.4829459190368652, -0.08124291151762009, -0.2689715027809143, 0.9023458361625671, 0.1976896971464157, 0.18980273604393005, 0.3135933578014374, 0.5913222432136536, -0.5203600525856018, -0.0410221703350544, 0.5236085653305054, -0.2535751461982727, 1.0794154405593872, 1.368709921836853, -1.7851449251174927, 0.42704159021377563, -0.1031675860285759, 0.6817436218261719, -0.18088223040103912, 0.8049570918083191, 0.08948411047458649, 1.9287855625152588, -0.2738359570503235, 0.10805225372314453, -0.7240666151046753, 0.4952975809574127, -0.520302951335907, -0.7584801316261292, -0.6663097739219666, -0.48398303985595703, -0.24722398817539215, 1.588961124420166, -0.5199193358421326, -0.8924790620803833, 0.21031902730464935, 0.8599275350570679, 2.112638235092163, -0.2542944848537445, 0.6824126243591309, -0.08865989744663239, -0.5297718644142151, 0.947668194770813, 0.15492863953113556, 1.3349769115447998, 0.029397178441286087, -1.79828941822052, -0.32611313462257385, 0.8774210214614868, 0.39338335394859314, 0.46272942423820496, 0.0854809433221817, -0.5748754143714905, 0.4467436671257019, 1.6525033712387085, -0.7304020524024963, -0.4535764753818512, -0.9279397130012512, 1.5468552112579346, -0.330756276845932, 0.7114301919937134, 0.3833301365375519, 0.6214118599891663, 1.1109745502471924, 0.6653859615325928, -1.2204495668411255, -1.2122936248779297, 1.1233662366867065, -1.2650634050369263, 2.398705005645752, -0.9410441517829895, 1.682780385017395, 0.30744409561157227, -1.233931064605713, -1.224339246749878, 0.7595686912536621, 0.504247784614563, -0.655708909034729, -1.392014980316162, 1.678949236869812, 0.5530781745910645, -0.4841843843460083, -0.31699472665786743, -0.186217799782753, 1.2814586162567139, 1.2500752210617065, 0.3004990518093109, 0.3991214632987976, 0.3029078245162964, -0.27147236466407776, 0.1122775673866272, -1.1647998094558716, -0.34116387367248535, -0.1999780535697937, -0.5571506023406982, 1.870590329170227, 0.38715898990631104, -0.7245387434959412, -0.03664012625813484, -0.08771509677171707, -0.01212905440479517, -1.580183744430542, -1.0841608047485352, -2.655102491378784, -0.7208299040794373, 0.6709150671958923, 1.0383356809616089, 1.0913474559783936, 1.3376761674880981, -0.3290189206600189, -0.15563525259494781, -0.0816969946026802, -0.42060360312461853, -1.3192754983901978, -0.23338162899017334, 0.6451345086097717, 0.3608875870704651, 0.2960107624530792, -1.772657036781311, -0.9691904783248901, 1.1686369180679321, 1.008975625038147, -0.62740558385849, 1.1960949897766113, 1.3135638236999512, 1.2133879661560059, -1.1603423357009888, -1.6710116863250732, 0.76377272605896, 0.09504615515470505, -2.3567867279052734, 1.232767105102539, -0.29148808121681213, 0.5235976576805115, 0.32805249094963074, 2.3495588302612305, 0.21717588603496552, 1.0190743207931519, 0.3594478964805603, -1.9056211709976196, -0.9837677478790283, 2.1166977882385254, 0.8828750252723694, -0.5578501224517822, 0.8914021253585815, -1.3778128623962402, -0.6111611127853394, -0.02310388907790184, 0.2560429871082306, 0.9695426821708679, -0.4573891758918762, -0.9313228130340576, 0.1684476137161255, -1.0282763242721558, 0.5946800112724304, -1.0766384601593018, 0.8708614706993103, -1.6012769937515259, -1.0669221878051758, -0.3578764498233795, -0.6790626049041748, -1.7209669351577759, 0.7030101418495178, -0.7968675494194031, -1.0640112161636353, -1.2878729104995728, -0.5015554428100586, 0.38256779313087463, -0.8078380227088928, -0.3621932566165924, 0.10502742975950241, 0.8446518182754517, 0.6937357187271118, 1.4491016864776611, -1.6303132772445679, -0.77444988489151, -0.8012405037879944, -0.9514670372009277, -0.42701029777526855, 0.027749061584472656, -0.3125731348991394, -1.7203255891799927, 0.1345442533493042, -0.3911517560482025, -0.2024035006761551, 0.027244409546256065, -2.9887290000915527, 2.569817543029785, 0.8786414265632629, -0.29413947463035583, -1.4351592063903809, -0.012248034588992596, -0.2588725686073303, -0.1583881825208664, 0.6239986419677734, 0.8205341100692749, 1.7084494829177856, -0.40084871649742126, -0.3098575472831726, -0.4516938030719757, -0.7799155712127686, 1.8744137287139893, -1.1491819620132446, -1.3895535469055176, -0.006219587754458189, -0.35271158814430237, 1.2078551054000854, -0.9740477800369263, -0.5119307637214661, 1.6051154136657715, 0.5291457176208496, -0.9980940222740173, -0.11221478879451752, -0.18508777022361755, -0.015889877453446388, 0.1559184044599533, -0.3437545895576477, -0.42962169647216797, 0.4011400043964386, -1.5420747995376587, 0.7564927339553833, 0.5368254780769348, 0.7720270156860352, -0.05137905478477478, 0.2727476954460144, -1.5222622156143188, -1.3648300170898438, -1.4036608934402466, 0.9895349740982056, -2.6763176918029785, 0.7428247332572937, -1.1110097169876099, 1.2127830982208252, -0.6208186149597168, 0.46353527903556824, 0.8710590600967407, -1.155694842338562, 1.365421175956726, -0.7788193821907043, 0.9478315711021423, 0.04546497389674187, 1.045127272605896, -0.2645995318889618, 0.4404478967189789, -1.5374476909637451, -1.4383350610733032, -2.0580639839172363, -0.4472264349460602, 1.4934083223342896, 1.1426154375076294, -0.8259202241897583, -0.08623570203781128, 0.011714883148670197, -0.7236521244049072, 1.1980297565460205, 1.1000912189483643, -0.890002965927124, 1.3131332397460938, -0.7005836963653564, -0.020848048850893974, -0.4348485469818115, 0.3702312707901001, 0.6881623268127441, 1.6442031860351562, -0.025289718061685562, 0.22888948023319244, 2.1260108947753906, -0.8661059141159058, 0.9951624870300293, -0.46394190192222595, -0.7491205930709839, -0.6870468854904175, 0.7439974546432495, 1.4038457870483398, 0.5305868983268738, -0.2247006744146347, -0.37501561641693115, -0.3242291808128357, -0.10226147621870041, -0.4593457281589508, 0.08156553655862808, -0.680750846862793, 0.14584504067897797, 0.21600031852722168, -0.2849968373775482, -1.0847227573394775, -0.23888255655765533, 2.1151862144470215, -2.77606201171875, 0.947589635848999, 0.7223991751670837, -0.2911135256290436, 1.0288115739822388, -1.1864005327224731, -1.7656757831573486, -1.3407542705535889, 0.626666247844696, 0.7819126844406128, 0.029396040365099907, 0.12759743630886078, 0.9234277009963989, -0.23197944462299347, 1.2979074716567993, -2.177598714828491, 0.40807241201400757, -0.6489871740341187, 0.9629749655723572, 0.5465651750564575, -0.4438037574291229, 0.9077268242835999, -0.22961848974227905, 0.7486611008644104, -0.9556559324264526, 0.33723732829093933, -0.4738191068172455, 1.3324694633483887, 0.6657416224479675, -0.3108365535736084, 0.41411006450653076, 0.21452787518501282, -1.0870144367218018, 1.5016591548919678, -0.6290556788444519, 0.055634260177612305, -0.2724192440509796, -0.5768334269523621, -0.6267091631889343, -0.1726653128862381, 0.8251093029975891, -0.18872371315956116, -0.6313266158103943, -1.4810329675674438, -1.1858989000320435, -0.06117523834109306, 1.1997926235198975, 0.2583635449409485, 0.5208132266998291, -1.9676814079284668, -1.6355690956115723, -1.2408301830291748, -1.4844024181365967, -1.1480679512023926, 0.06744211912155151, 0.7055112719535828, 0.7416011095046997, -0.38057616353034973, 1.2104450464248657, -0.2415645867586136, 0.2389775812625885, 0.15589410066604614, 0.30810651183128357, 1.374988317489624, -1.408028244972229, 0.6566829681396484, 1.2311551570892334, 0.7450230717658997, 1.2589573860168457, 1.7868393659591675, -0.12354893237352371, -1.215919852256775, 0.04243357479572296, 0.6581975221633911, -0.5196390151977539, 0.3285712003707886, -1.1684638261795044, 1.583858847618103, -1.1496610641479492, 0.853967010974884, 1.3197486400604248, -0.7882124185562134, -1.605812907218933, 0.18393363058567047, -1.8155407905578613, -0.9178430438041687, -0.6925777196884155, 1.1437759399414062, -1.1832375526428223, 0.08495321124792099, 0.605235755443573, -1.7882827520370483, 0.6553471684455872, 1.805000901222229, -0.2518637478351593, -0.10262323170900345, 0.5477827787399292, -1.14608895778656, -0.6241768002510071, -1.6944808959960938, -0.630171537399292, 0.5002231597900391, -0.40419167280197144, -0.5636353492736816, 1.4417147636413574, 1.3753074407577515, 0.18750588595867157, 1.0597271919250488, -0.49529653787612915, 1.0097324848175049, -1.475753664970398, -2.494051456451416, 0.6498326659202576, -0.8316230773925781, 0.267221599817276, -0.5520009398460388, 0.08389702439308167, -0.17789432406425476, -0.3159184753894806, -0.9373629093170166, 0.09176930040121078, -0.32127368450164795, -1.6865490674972534, -0.16523855924606323, 2.2335402965545654, 1.6369255781173706, 0.6415473222732544, 1.0595080852508545, -0.8263913989067078, 0.36039572954177856, 0.2699894309043884, -2.347676992416382, 0.9988788366317749, -0.008000964298844337, 0.4260372817516327, -1.5727057456970215, -0.11945100873708725, 0.5936875939369202, 0.3421490788459778, -0.5134766697883606, 0.20148655772209167, -0.7941969633102417, -0.04521522670984268, -0.6338891983032227, -0.48390257358551025, -0.385606974363327, -0.9373287558555603, -2.180870532989502, -0.15274877846240997, 0.13378113508224487, -0.5714137554168701, -0.21059556305408478, -1.216785192489624, 0.3663784861564636, -0.17967233061790466, 1.8661606311798096, 0.031234867870807648, -0.3943692445755005, 0.42509567737579346, -0.30295589566230774, 1.3341002464294434, 1.365883231163025, -0.8226155638694763, -1.0250840187072754, -0.17976467311382294, -0.7869089841842651, 0.9752145409584045, 1.1509287357330322, 0.2368965595960617, -1.2462897300720215, 2.3066117763519287, 1.879077672958374, -1.8469306230545044, 0.030749740079045296, -1.0163744688034058, -0.9636075496673584, 1.159708857536316, 0.987687885761261, -0.7041213512420654, -0.3088289201259613, -0.08263590186834335, 0.09946639090776443, 0.3282065987586975, -1.7897650003433228, 0.2721555233001709, 0.03568873181939125, 0.7648050785064697, -0.42800241708755493, 0.8954304456710815, -1.6361745595932007, -0.2949364483356476, -0.16808101534843445, 0.3760291337966919, -0.6414823532104492, 0.8307330012321472, 0.3003499209880829, 1.1191829442977905, 0.24527083337306976, 0.35602644085884094, 0.9154325723648071, -1.7583054304122925, 0.04979366064071655, 1.055419683456421, 0.04921531677246094, 0.3299960494041443, -0.0810171589255333, 0.46059277653694153, 0.7154347896575928, 0.587039053440094, 1.069520115852356, 1.0624682903289795, -1.2741020917892456, -0.5349624752998352, 0.25849488377571106, 0.1402476578950882, 0.670776903629303, 0.6961838006973267, 0.6771214008331299, -0.05941889435052872, 0.6787862181663513, -1.3571590185165405, 0.6024125218391418, 0.8911147117614746, -0.5105774998664856, 0.8206746578216553, -1.2033799886703491, -1.3357514142990112, -0.6922658085823059, -1.4972920417785645, 0.24199947714805603, 0.9637387990951538, -0.12313321232795715, 0.4562104046344757, -0.6864912509918213, 1.196526050567627, 0.8812158107757568, 2.1664059162139893, -0.2704767882823944, -0.7225445508956909, 0.46061843633651733, -1.3412137031555176, -1.8861885070800781, 0.08319756388664246, -1.0256905555725098, 1.0260018110275269, 1.3358776569366455, -0.8492098450660706, -1.1774578094482422, -0.3852396011352539, -0.2389589548110962, 1.4304803609848022, 0.8453549146652222, -0.8826238512992859, 0.5879193544387817, -0.9556878209114075, -0.6446486711502075, -0.04813605919480324, 0.16449612379074097, 1.393050193786621, 0.3621852993965149, 0.1842755377292633, -1.3884458541870117, -0.3981612026691437, 0.36595484614372253, -1.309308409690857, -0.4592064917087555, -0.09911197423934937, -1.054349422454834, 0.9073368906974792, 1.0646193027496338, -0.0916282907128334, -0.13304230570793152, 1.9178931713104248, 2.1331090927124023, -0.1608593463897705, 0.7071466445922852, -0.9403007626533508, 0.10056960582733154, -1.38163423538208, 2.467808961868286, -0.9178617000579834, 0.9479462504386902, 1.0210682153701782, 0.8690661191940308, 0.9760029911994934, -0.1900390237569809, -0.8373558521270752, 1.0693564414978027, -3.300529956817627, -0.3033026456832886, -0.6163977384567261, 0.07854921370744705, -0.01422160305082798, -0.5382474064826965, 1.2608469724655151, -0.23871703445911407, 0.2516551911830902, -1.4078962802886963, 0.17807798087596893, -0.27951499819755554, -0.32470375299453735, 0.33708739280700684, -0.3905602693557739, -1.312508225440979, 0.2721131145954132, -0.5276113152503967, 0.5430166721343994, -0.3155907988548279, 1.4220010042190552, -0.2442438304424286, -1.1777441501617432, -0.04652144014835358, 1.2724478244781494, 0.7555330991744995, -0.48357635736465454, 0.9357997179031372, -0.9724538922309875, -0.5726158022880554, -0.23374763131141663, -0.3088597357273102, 1.5067554712295532, -0.44115036725997925, 0.666394054889679, 0.6112034320831299, -1.0651698112487793, -2.3573052883148193, -0.8840983510017395, -2.2075319290161133, 1.7622792720794678, -0.8269596695899963, -0.39005643129348755, -0.07548932731151581, 1.595874547958374, 0.07304126024246216, 0.2513958513736725, -0.2832564115524292, 0.8068016767501831, -0.6662847399711609, -0.8335273265838623, 0.27034181356430054, 1.2098103761672974, 0.10006467998027802, -0.12817339599132538, 0.007595733739435673, 0.5829476714134216, 1.5815542936325073, 2.3297932147979736, 0.40067926049232483, 0.16518062353134155, 2.2534401416778564, -1.0515589714050293, 0.9380159378051758, 0.15660491585731506, -0.13244499266147614, -0.9371905326843262, -0.336055189371109, 1.7380306720733643, 0.586528480052948, -0.058436159044504166, 1.064992070198059, 1.1412822008132935, 0.0757758840918541, -2.386169672012329, 0.8686137795448303, -0.8065719604492188, -0.438729465007782, -0.04139725863933563, 1.702532172203064, 0.6513054370880127, -0.4932388961315155, -0.6731023192405701, -0.8793643116950989, -0.8263064622879028, 0.3928627371788025, 2.798593282699585, -0.8313782215118408, -0.8102985620498657, -0.1559729129076004, -0.8990676403045654, -0.790658175945282, 2.1733601093292236, 0.049533650279045105, 0.5475041270256042, -0.5611205697059631, -0.389120489358902, 0.6444801688194275, 1.8676526546478271, 0.026980701833963394, 0.2531053125858307, -0.2115623950958252, 0.5194998979568481, -0.9566535353660583, 1.7743732929229736, 0.15556542575359344, -1.307216763496399, 0.0978349968791008, 0.08064761012792587, 0.8377115726470947, 0.7037152051925659, -1.355384349822998, 2.2607030868530273, 0.9103236794471741, 1.511324167251587, -0.6105698347091675, -0.4392581880092621, 1.9121547937393188, -0.3870187997817993, 0.023037200793623924, 0.6484507322311401, -1.5425242185592651, -0.6014519333839417, -0.05506999045610428, 0.6390459537506104, 1.443367600440979, -0.10153012722730637, 0.9676790833473206, 0.9865692853927612, -0.6903195977210999, -1.5518324375152588, -0.5020042657852173, -0.5473311543464661, -0.7586451172828674, 0.25078433752059937, -0.6742576360702515, -0.8973867297172546, -1.7638013362884521, 0.6888120174407959, 2.3425114154815674, 0.16525974869728088, -0.8453882336616516, 1.424997091293335, 0.8614158034324646, -1.291101098060608, 0.8368922472000122, 0.5326431393623352, 0.5644975304603577, 0.6913362145423889, -1.4640637636184692, -0.7896548509597778, -2.302825450897217, 0.4507652521133423, 0.7154748439788818, -0.6167116761207581, 0.14913012087345123, -0.6927317976951599, -1.0454479455947876, 0.14425407350063324, 1.2143803834915161, 0.280184805393219, -0.37767839431762695, -0.3787553310394287, 0.09256716072559357, -0.09625080227851868, -1.1352944374084473, -0.3994854986667633, 0.5867871642112732, 1.0096631050109863, -0.23575268685817719, -0.5329071283340454, 0.49258965253829956, 1.1984413862228394, 0.23490525782108307, 0.7465758919715881, -0.7008439898490906, -1.1741995811462402, 0.07016807049512863, 1.578970193862915, 1.1442161798477173, 0.24593204259872437, -0.6869587302207947, -1.402497410774231, -0.7321228981018066, -0.8127636313438416, -0.7881033420562744, -0.3990018367767334, 0.30059483647346497, 0.8321678638458252, 1.0475977659225464, 0.5303555130958557, -0.39766985177993774, 0.9681347012519836, -1.0251564979553223, -0.7972730398178101, -0.5622841119766235, 0.1665159910917282, -0.0647801011800766, 0.27994605898857117, 0.03469729423522949, -0.27462702989578247, -0.2451181262731552, 1.254157543182373, -1.0348402261734009, -1.498887300491333, -2.0904815196990967, 0.39133262634277344, -2.3819591999053955, -0.9347224235534668, 0.17240197956562042, 1.4363141059875488, 1.1416457891464233, -0.3222225308418274, -0.6731452345848083, -0.5333107113838196, 0.03261173889040947, 0.055538587272167206, -0.3899114429950714, -2.556640148162842, 0.47004419565200806, -1.1465890407562256, -0.8101454973220825, 2.6027705669403076, 0.5141388773918152, -1.2699229717254639, 0.07174313068389893, 0.37850266695022583, -0.11119713634252548, -2.4477016925811768, -0.3560769259929657, -1.360082745552063, -0.19574880599975586, -0.6362502574920654, 0.4726622402667999, 0.37321531772613525, 1.344241738319397, -0.7322660088539124, -1.3158663511276245, -2.7648298740386963, -1.1343461275100708, 0.40926530957221985, -1.1500436067581177, 1.5286824703216553, 0.4685802757740021, -0.0505598708987236, -0.3008743226528168, -0.6678045392036438, -2.1037213802337646, -0.1845199316740036, 0.5891622304916382, 0.8770216703414917, -1.001751184463501, 1.6901203393936157, 0.191898375749588, 0.771995484828949, 1.6346087455749512, 2.1920278072357178, 0.15525056421756744, 0.2533501088619232, 0.12203627824783325, -0.10647349059581757, -1.1516607999801636, -0.19078075885772705, 1.110794186592102, 1.0147162675857544, 0.4198545813560486, -0.1372828185558319, 0.7278899550437927, -1.1405611038208008, 0.5545917749404907, -0.9706957340240479, -0.45411789417266846, 1.2261251211166382, -1.46273934841156, -2.2049672603607178, -1.9830477237701416, 1.327475666999817, -1.494701862335205, -0.9193080067634583, 0.6010003685951233, 0.12867435812950134, 0.8697535395622253, -0.43218743801116943, -0.5726337432861328, 0.44782450795173645, 0.8754544854164124, 0.36013063788414, -1.0268696546554565, -0.6465502381324768, -0.7422588467597961, -1.7048687934875488, -2.0649850368499756, 0.29514575004577637, -2.248933792114258, -0.4876515567302704, 0.0976027324795723, 1.9536938667297363, 1.0053868293762207, 0.9804271459579468, -0.2978890836238861, 0.3176785707473755, -0.4293004274368286, 0.869560718536377, 1.12491774559021, -1.153061032295227, -0.1233539953827858, -0.2558138072490692, -1.4154006242752075, 0.2544127106666565, -0.8339918851852417, 0.04529238119721413, -0.6683336496353149, -0.28116118907928467, 1.1858326196670532, 0.2095479518175125, 0.6563934087753296, -0.03124265745282173, -0.7573772668838501, -0.3804745674133301, 0.9643255472183228, 0.9006009101867676, -1.240047574043274, 0.6830062866210938, 1.4402998685836792, 0.26089856028556824, -0.09679249674081802, -0.6263400912284851, -0.35005050897598267, -1.3847850561141968, 0.42981454730033875, -0.828668475151062, -0.2941567897796631, -0.06181848794221878, -1.7551368474960327, -0.7778016328811646, 0.6513954997062683, 1.2020810842514038, -0.9251306056976318, -0.15850678086280823, -0.7554669380187988, 0.953087329864502, -1.2380073070526123, 0.5127635598182678, 0.01618744246661663, 1.560341715812683, -2.595682144165039, -1.0206211805343628, 1.665369987487793, 0.5126082897186279, -1.1057716608047485, 0.08085926622152328, -0.5803529620170593, -1.0807762145996094, 0.5367636680603027, -0.31083112955093384, 0.6609829068183899, 0.3794691860675812, 1.8378843069076538, -0.7589308619499207, -0.028207050636410713, 0.2051600217819214, 1.819603681564331, 0.4141826331615448, 0.35748839378356934, -0.6707932353019714, -2.1577751636505127, -0.6667338609695435, -0.8709536790847778, -0.5385633707046509, -1.033764362335205, 0.3543485701084137, 0.9750017523765564, 1.3881827592849731, -1.2560837268829346, 0.08721674978733063, -1.2945005893707275, 0.8668912649154663, 0.8668495416641235, 0.3456573486328125, -0.5148473381996155, 0.1094549372792244, -0.7874205112457275, 0.39797529578208923, -0.006850156467407942, 0.3796825408935547, -0.331145703792572, -0.4099688231945038, -0.49604466557502747, 1.1819175481796265, 0.3641407787799835, 1.2216888666152954, 0.7057058215141296, -0.47574156522750854, -1.3980093002319336, -0.22010526061058044, -0.1046365275979042, -0.7466110587120056, 1.8494620323181152, -1.5774372816085815, -0.7656967639923096, -0.6093950867652893, 0.33887630701065063, -0.1367558389902115, -0.5477283596992493, 0.22286637127399445, 0.11335372179746628, -0.659254252910614, 1.3797560930252075, 1.2906032800674438, 0.5265167355537415, -0.15733979642391205, -0.19366386532783508, -1.3107682466506958, -0.26293957233428955, -0.8911560773849487, 0.17452555894851685, -0.28447026014328003, -0.6401796936988831, 0.3438458740711212, -0.027993094176054, 1.3271781206130981, 0.255569726228714, 0.5462062358856201, 1.7710177898406982, 0.27667608857154846, -0.2291002869606018, -0.2434689849615097, -2.5153205394744873, -1.6758695840835571, 0.7411133646965027, -1.7381701469421387, 0.9058426022529602, 1.1622425317764282, -0.9319484829902649, -1.1635396480560303, 0.09770143032073975, -0.4194422960281372, -0.29110801219940186, 1.1443852186203003, -0.32407093048095703, 1.3475862741470337, -0.5536256432533264, -1.3044357299804688, 0.8071374297142029, -0.08254233747720718, 0.8590745329856873, -0.7008742690086365, -0.2870863676071167, 0.004550641402602196, -0.9428781867027283, 0.046648070216178894, 0.5417793393135071, 0.08152086287736893, -1.4173805713653564, -0.7304826378822327, -0.02768053114414215, -1.0148030519485474, -0.7164207696914673, 0.8661661744117737, 0.6558787822723389, 0.43397048115730286, 1.3102244138717651, 1.3056029081344604, -1.1420602798461914, 0.1676895171403885, 1.6133726835250854, -1.751954197883606, -0.09901528060436249, -0.9986785650253296, 0.24564537405967712, -0.20503360033035278, -0.691156268119812, 0.5678089261054993, -0.30098745226860046, 0.44544294476509094, 0.7213132977485657, -0.11909625679254532, 0.6061345338821411, -1.3724052906036377, -0.36868298053741455, -0.7643720507621765, 0.5736678242683411, -1.8386205434799194, -0.25544705986976624, -0.3944190740585327, -0.9091652631759644, -2.07011342048645, 0.7630968689918518, 1.273546576499939, -0.20034745335578918, -0.6324070692062378, -0.635785698890686, -0.13334737718105316, 0.30491477251052856, 0.36678802967071533, 0.9549075365066528, -0.01681622862815857, 0.433149516582489, 1.1200097799301147, 0.2956520617008209, -0.13629373908042908, -1.5331377983093262, -0.3879631757736206, -1.1321651935577393, -0.28787761926651, 0.6440752148628235, -0.2327948659658432, 0.04425318166613579, -1.182605504989624, 1.2469102144241333, -0.8520402908325195, -0.21552743017673492, -0.0829477608203888, -0.7270567417144775, -1.137987732887268, -1.1933045387268066, 0.45054349303245544, -0.4911215007305145, 0.8967333436012268, 0.9016598463058472, 0.8277237415313721, -0.6531859040260315, 2.2023255825042725, -0.1793837696313858, 1.4749031066894531, 1.522505521774292, 1.2429569959640503, -1.774734616279602, -0.10192003101110458, 1.6739661693572998, 0.7432459592819214, 0.14017701148986816, 1.2702646255493164, -0.7643227577209473, 0.3426227271556854, -0.46204856038093567, 0.34235644340515137, 0.2087295651435852, 1.5983482599258423, 0.04637717455625534, 0.7551071047782898, -0.18471968173980713, 1.1441593170166016, 0.17709819972515106, 1.0641770362854004, -0.23255902528762817, -0.190013587474823, -0.8339852690696716, 0.5068429112434387, -2.007924795150757, 0.325798362493515, 1.0772926807403564, -0.20307675004005432, -0.9432703256607056, 0.04185265302658081, 0.6969090700149536, -0.040494464337825775, -1.7927595376968384, -0.40194112062454224, -0.0531131774187088, 0.3569624722003937, -1.0672698020935059, 0.17805004119873047, -0.7153244018554688, -0.8680130243301392, -2.2597854137420654, 1.6836153268814087, -1.1444692611694336, -0.6269059777259827, 0.5502830743789673, -1.1219497919082642, -1.3708057403564453, 0.42911890149116516, 1.3833041191101074, 0.503324031829834, -0.8322871327400208, 1.185744285583496, 3.012946367263794, 1.4718563556671143, 3.425023078918457, -0.27467089891433716, -0.32126566767692566, -0.8444104790687561, 1.0410150289535522, -1.4838311672210693, -0.2344060093164444, 0.278425931930542, -0.37456849217414856, 0.6607451438903809, -0.15466323494911194, 1.104171872138977, -1.816017985343933, 0.21399782598018646, -0.1517079770565033, 0.7219954133033752, -2.2842066287994385, -1.4656397104263306, -0.9618842601776123, 0.4543468654155731, -0.35222163796424866, 0.29490792751312256, 0.1703430414199829, -0.14909277856349945, 1.9357627630233765, -0.6134451627731323, 0.9634491801261902, 0.3998086452484131, -1.8008531332015991, 0.5130555033683777, -0.30323368310928345, 0.3157743513584137, 0.7664967775344849, -0.5220826268196106, 0.15244358777999878, -1.1370484828948975, 0.01771167851984501, 0.2415597140789032, -1.0722665786743164, 0.8111208081245422, -1.3887487649917603, -0.2589159905910492, -0.5504198670387268, 0.6430971026420593, 0.21280935406684875, 1.8437249660491943, -0.7298426032066345, -0.5448675155639648, -0.8936702609062195, -0.34691378474235535, 0.7702645063400269, -0.19546590745449066, -0.28526392579078674, 0.8188953399658203, 0.5356029272079468, -2.234575033187866, 0.3974209129810333, 1.1415975093841553, -1.302061676979065, 0.12402298301458359, 1.2094271183013916, 0.8347518444061279, -0.7462936639785767, -0.2812917232513428, -2.4187285900115967, -1.4249619245529175, -1.2080727815628052, -1.0401455163955688, 1.1719326972961426, 0.33868175745010376, -2.118718385696411, 0.24394090473651886, 0.28757479786872864, 0.1907263994216919, -0.6046323776245117, -0.8422470092773438, -0.8048197031021118, 0.3535330593585968, 1.9179977178573608, 1.3415604829788208, 1.0707321166992188, 0.8452953100204468, 0.8589584231376648, 1.198682188987732, -0.87424635887146, 1.9178292751312256, 0.43505364656448364, -0.3570357859134674, 0.21522949635982513, -0.46098455786705017, -0.400718092918396, 0.11791298538446426, -1.2301552295684814, 0.22072595357894897, 1.8776606321334839, 0.8472408056259155, 0.6469457149505615, 0.452698677778244, -2.4139201641082764, 0.173808753490448, -1.617616891860962, -0.23848210275173187, 0.6423895359039307, -1.681473970413208, -0.1373465359210968, -0.4258417785167694, -1.7823656797409058, -0.020922349765896797, -0.19134104251861572, -0.3606749176979065, -0.17360977828502655, -0.406676322221756, 0.29244714975357056, 0.43652427196502686, -0.920408308506012, 0.9520158767700195, 2.146790027618408, -1.075901746749878, -0.009020937606692314, -0.32194748520851135, -2.1141715049743652, -0.7342143058776855, 0.30514881014823914, 1.5788854360580444, 0.402360737323761, -0.030224910005927086, 0.3953511416912079, 0.7577989101409912, 1.3093559741973877, -0.471515417098999, -1.9213380813598633, 0.26233014464378357, -0.45909255743026733, 0.911529541015625, -0.8331479430198669, 0.193854421377182, -0.6256781816482544, 0.4275307357311249, 1.1557785272598267, -1.1586225032806396, -0.3535076677799225, 1.4791688919067383, 0.27464598417282104, -0.3742831349372864, -1.0349584817886353, -1.5020431280136108, 0.5661960244178772, -1.342272400856018, 0.22869634628295898, 0.4726131558418274, 1.8658298254013062, -0.29708132147789, -0.3951442539691925, 0.5597388744354248, 1.092046856880188, 0.7011576294898987, -0.48321011662483215, 0.5559635758399963, 0.8140822052955627, -0.1345798224210739, 0.16365386545658112, -0.3281737267971039, 0.965745210647583, 0.5529528856277466, -0.6285428404808044, -1.2099804878234863, -0.08316019922494888, 1.014547348022461, 0.7405785918235779, 0.05175989866256714, 0.5838534832000732, 0.860569417476654, 0.9357192516326904, -1.5074079036712646, 1.5262020826339722, -0.9447537660598755, -0.2745864689350128, -0.4235137403011322, 1.6238731145858765, -2.4171674251556396, 0.12474723905324936, 1.7580064535140991, 1.7730534076690674, -0.3509052097797394, -0.37012067437171936, -0.5522492527961731, 0.6126425862312317, 0.024498267099261284, 0.2407161146402359, 1.5439156293869019, 1.1058669090270996, -0.9019353985786438, -2.057931661605835, 0.08885043114423752, -0.7361693978309631, -0.01676071435213089, -0.44009190797805786, -0.9663522839546204, 0.7862975597381592, -1.3484760522842407, 0.20270901918411255, -0.25448399782180786, 1.005225419998169, 0.9169512987136841, -1.958595871925354, 0.7877666354179382, 1.6104477643966675, -3.308802843093872, 1.4770222902297974, -0.3093443810939789, 1.4724502563476562, 0.7402892708778381, -0.11243359744548798, 0.14953471720218658, 0.6142089366912842, -0.8148166537284851, 0.7539476752281189, 1.1602637767791748, 1.097547173500061, 0.6212635636329651, 0.7227308750152588, -0.886083722114563, 1.288862943649292, -1.1680803298950195, -0.8945519924163818, 0.1650741845369339, 0.5656319856643677, 0.9048743844032288, -2.2960915565490723, -0.4185231626033783, 0.24497359991073608, -1.3162423372268677, -0.7167928218841553, -0.033617302775382996, 0.2656024694442749, -0.44960299134254456, 0.020817138254642487, -1.3421519994735718, -0.09460314363241196, -0.8570413589477539, 1.7727179527282715, 1.3479076623916626, -0.09592019766569138, 0.977760374546051, 0.3345082700252533, 0.38438835740089417, -1.9677059650421143, -0.6238499283790588, -0.4645140767097473, -1.784682273864746, -0.16298112273216248, 2.2931482791900635, -2.022881031036377, 0.06187822297215462, -1.7025880813598633, 0.11943265795707703, 1.183100700378418, -0.21920470893383026, 0.9906466007232666, -0.6017428040504456, -1.4273898601531982, 0.5463981628417969, -1.490441918373108, -0.48946231603622437, -0.00974551122635603, -0.41727039217948914, 0.24702616035938263, -0.28529176115989685, 1.3661338090896606, 0.9209854006767273, -1.634640097618103, -0.7233976125717163, -0.6896048784255981, 0.2837883234024048, 1.0921332836151123, -1.3210629224777222, -0.6138050556182861, -0.5137700438499451, -0.775318443775177, -0.15198136866092682, -0.7915181517601013, 1.987831711769104, 0.8558945655822754, 0.13108550012111664, 0.7154549360275269, -1.2629855871200562, -0.5401704907417297, -1.6495745182037354, -0.8994823694229126, 0.7860240340232849, -0.3619561195373535, 0.8015148639678955, -0.8421728610992432, -2.5264649391174316, 1.8284080028533936, -1.5828046798706055, 0.6014090180397034, -0.09684887528419495, 0.646581768989563, 1.1799345016479492, 0.23548266291618347, -1.3086835145950317, -0.1818990707397461, -0.19806458055973053, 0.4224678874015808, 1.3566334247589111, 0.2948582172393799, -0.5630946755409241, 0.4431559145450592, -0.8760425448417664, -1.8409231901168823, 1.6686755418777466, -1.9888797998428345, -0.9492393732070923, 0.8041999936103821, -0.9074183702468872, 0.4192031919956207, 0.6156699657440186, 0.6282875537872314, 0.4236145615577698, -1.0100723505020142, -1.4109045267105103, -0.10751467198133469, -1.172782063484192, 0.3232298791408539, 1.2512023448944092, -0.9277379512786865, -2.0981905460357666, -0.3625198006629944, 2.395838499069214, -1.0871785879135132, -0.22912518680095673, -1.2835321426391602, 0.13380999863147736, -1.0333075523376465, 1.3702564239501953, -0.6732354760169983, -0.8512462377548218, -1.4320106506347656, -1.8094096183776855, -1.0070769786834717, 0.09046242386102676, -0.3108370900154114, -0.4088086485862732, 0.7994580864906311, -0.47185787558555603, -1.963444471359253, -1.2447978258132935, -1.1577600240707397, 0.3411812484264374, -1.477197289466858, 1.2873867750167847, 0.959495484828949, 0.224787637591362, -0.22477272152900696, 0.9430710077285767, -1.260241985321045, 1.082265853881836, 1.889603614807129, -0.41244223713874817, -1.1925690174102783, 0.06951901316642761, -1.2318086624145508, 2.09118914604187, -1.1999112367630005, -0.10112233459949493, -0.7520352005958557, 0.3260095417499542, -1.6522899866104126, -1.297008752822876, -1.2793828248977661, 0.9424965977668762, -1.2305781841278076, 1.7701785564422607, -0.49499452114105225, -0.5504827499389648, 0.6927633285522461, -0.5268692970275879, -1.5071547031402588, -1.3288967609405518, -0.3790654242038727, 0.4782463312149048, 1.6599128246307373, -0.050496526062488556, 0.31438544392585754, 1.2116773128509521, -1.883984088897705, 0.506316602230072, 0.02817363291978836, -0.67762690782547, 0.3156580924987793, 0.8965408802032471, 1.1028547286987305, -1.6489500999450684, -1.8623346090316772, 0.48705992102622986, -0.2611792981624603, 1.050653100013733, -0.12545348703861237, 1.421971321105957, 0.14396291971206665, -1.5116957426071167, 0.40946364402770996, 0.7648628354072571, -0.8525393605232239, -0.4282662570476532, -3.299060583114624, -0.3720266819000244, 1.1458513736724854, 0.6871222257614136, 1.2256157398223877, -1.2497144937515259, 1.117648959159851, 0.007661200128495693, -0.6688439249992371, 0.13749413192272186, 0.1220078393816948, -0.5981660485267639, 0.844821572303772, 0.712533175945282, 0.973362386226654, -0.41871339082717896, 0.10298275202512741, 0.18458779156208038, 0.05809686705470085, 0.3115381896495819, 0.13879138231277466, -0.9507129192352295, 1.42742919921875, 0.29192012548446655, 0.31453877687454224, 0.05294005200266838, -0.014566613361239433, 0.22253216803073883, 0.4459241032600403, 1.2351921796798706, 0.1683921068906784, 1.428023338317871, 1.5061898231506348, 0.35451221466064453, 0.393540620803833, 0.4607991576194763, 0.32425644993782043, 1.4326894283294678, 1.3849025964736938, 0.08886933326721191, 0.05015217885375023, -0.4648687541484833, -0.5348029732704163, -2.948493480682373, 2.135988235473633, -0.46360012888908386, 0.46310099959373474, -0.7626565098762512, -0.4047320783138275, 0.7327659130096436, 0.30698519945144653, 1.643436074256897, -0.41184327006340027, -2.056928873062134, 0.05901447683572769, -0.14355237782001495, -0.5113880634307861, -0.08546824008226395, -0.6112269759178162, -0.3150780498981476, 0.4718658924102783, -0.47564682364463806, 0.5665902495384216, 0.5084412097930908, -0.9028715491294861, -1.380678415298462, -0.34564265608787537, 1.5157400369644165, -0.11210376769304276, -0.6040621399879456, 0.14017754793167114, 0.18960130214691162, 1.9584875106811523, 0.6191452145576477, 0.41838207840919495, -1.3477708101272583, 0.9543505311012268, 0.8897458910942078, 0.2168336808681488, 0.03790609911084175, 2.4104573726654053, 0.5754581689834595, 0.3860636353492737, -0.0029064537957310677, 0.965831458568573, -1.0744682550430298, -0.14984983205795288, 1.2977731227874756, 1.035262107849121, 0.38382843136787415, 0.8477321863174438, -0.2775280177593231, -0.6692207455635071, 1.0533230304718018, 1.4218722581863403, 0.5114611387252808, 0.339449018239975, 0.3565402925014496, 0.4321534335613251, -0.7914818525314331, 1.0288374423980713, -0.7260706424713135, 1.0150667428970337, 0.18940652906894684, 0.4296051263809204, 1.1617283821105957, 0.7718927264213562, 1.202389121055603, 1.5348708629608154, 0.09731177985668182, 0.2766629457473755, -0.698051393032074, -1.0525394678115845, 0.7906367182731628, 0.9540837407112122, -0.761536717414856, -0.17992118000984192, -1.118444800376892, 0.16464026272296906, 1.1182572841644287, -0.480417937040329, 0.7110475301742554, 1.552201509475708, -0.7633139491081238, 0.40715983510017395, -0.5874559283256531, 0.1722734421491623, -1.1052396297454834, 0.029204759746789932, -0.7895840406417847, 0.7902141809463501, -1.425063133239746, -0.41839438676834106, -0.3357621133327484, -0.936916172504425, -0.3715650737285614, 0.6485233902931213, -0.5960349440574646, -3.4201221466064453, 1.9399915933609009, 0.5457891821861267, 1.992270588874817, -0.27270039916038513, 0.9452564716339111, 0.20903821289539337, -1.4832539558410645, 0.1406976729631424, 1.0244202613830566, -1.8370239734649658, 0.3581894636154175, -1.591971755027771, 1.2504777908325195, 1.2626354694366455, -0.23985080420970917, 0.5650430917739868, -0.064983069896698, 1.689527988433838, -1.060171127319336, 3.539546012878418, -0.44704848527908325, 0.6599339246749878, -2.3042473793029785, -0.8901151418685913, -0.09927035123109818, -1.1139140129089355, 2.060640573501587, 0.08819646388292313, 0.7837857604026794, 0.038679126650094986, -0.8571968078613281, -0.5327146053314209, -0.3893727660179138, -1.9698642492294312, -0.45324060320854187, -1.322333812713623, -0.44958987832069397, 0.6923103332519531, 1.8553208112716675, -0.7756953835487366, 0.14137686789035797, -0.2913273870944977, -1.4262187480926514, 0.03940246254205704, 0.6554354429244995, -0.13462892174720764, -0.4802751839160919, -2.0717716217041016, 0.34562286734580994, 1.0998858213424683, -1.5041351318359375, -0.22730684280395508, -1.1949048042297363, -0.5774477124214172, -0.8662787675857544, -1.3001419305801392, 1.1922510862350464, -0.22377081215381622, 1.0609474182128906, 1.1913152933120728, -0.19194477796554565, -0.6620754599571228, 0.13798022270202637, 0.6241751909255981, 0.9149166345596313, -1.7362070083618164, 0.5049247741699219, -0.2034149020910263, 0.8654125332832336, -0.38106071949005127, 0.1546630859375, -1.0004222393035889, -0.521281361579895, -0.5007829070091248, -1.3143837451934814, -0.10632776468992233, -0.991004467010498, -0.8489006757736206, 0.1698686182498932, -1.4663382768630981, -0.2542538046836853, 1.3877159357070923, 0.44471025466918945, -0.30083024501800537, 1.9665641784667969, -0.24403713643550873, 1.0801318883895874, -1.5486325025558472, -1.6627589464187622, -0.7934297919273376, 0.8498479127883911, -0.8698151707649231, 2.1673078536987305, 0.3357035219669342, -0.6024683117866516, -0.8086227178573608, -0.5982629060745239, 1.291762351989746, 1.358680248260498, 1.395456075668335, -0.5408049821853638, -0.0494876503944397, 0.8380951285362244, -1.719204068183899, -0.15842944383621216, 1.038271427154541, 0.2236843854188919, 0.04718094319105148, -0.02134024165570736, -0.05144016444683075, 1.6671663522720337, -0.8771224021911621, 0.5394079089164734, -0.08141409605741501, -0.5154661536216736, -0.08323565125465393, -0.2689970135688782, -2.233441114425659, -0.861881673336029, -0.8093699216842651, -0.29527929425239563, -0.19014093279838562, -0.5636216998100281, 1.625514030456543, -0.2971140146255493, 0.03939250484108925, -0.9833841323852539, -0.19337429106235504, 1.3237996101379395, -1.1246024370193481, 0.4776623547077179, 0.23499836027622223, -0.7558451890945435, -1.9967995882034302, -1.537168025970459, 1.7630836963653564, -0.26502659916877747, -1.6086972951889038, 0.8485103249549866, 0.643271803855896, 2.1780712604522705, 1.4926704168319702, 1.0587878227233887, -0.8368819355964661, 2.0250754356384277, -0.12099503725767136, -0.2976444661617279, 1.7073235511779785, -0.4802381098270416, 0.1014215424656868, -0.272345632314682, 0.5835590958595276, -1.4094440937042236, -0.17406676709651947, -0.04667020961642265, 1.1900944709777832, -0.506483793258667, 0.31816375255584717, 1.5953478813171387, 0.4366419017314911, 0.10988779366016388, -0.6396816372871399, 0.046115484088659286, -1.5258477926254272, 1.9087893962860107, 1.1675910949707031, 0.08160468935966492, 1.0659071207046509, -0.6032538414001465, 0.6213553547859192, 0.0007147446740418673, 0.8625096678733826, -0.9952760338783264, 0.254995197057724, -0.10692283511161804, -1.7426881790161133, 1.5921404361724854, -0.5248655080795288, 0.9801254272460938, 0.7311399579048157, -1.3965449333190918, 0.9691985249519348, 0.3899707496166229, -0.10897205770015717, -0.9873964190483093, 0.05098346620798111, -0.4112074077129364, 1.2424653768539429, -1.8035249710083008, 0.40706050395965576, 0.4231444001197815, -0.25589874386787415, 2.2232978343963623, 1.320726752281189, -0.8042430877685547, -0.5782874822616577, -0.27214980125427246, -0.48014500737190247, 0.5511255264282227, 1.6200591325759888, -0.2006780505180359, -1.9330886602401733, -1.3078985214233398, -0.8701474666595459, -0.5845359563827515, 0.9644064903259277, -1.9733272790908813, -1.3210103511810303, 0.5948541164398193, 0.21616096794605255, -0.11111799627542496, 0.6115788817405701, -0.39434075355529785, -0.6742765307426453, 0.7369659543037415, 0.23955917358398438, 1.4506988525390625, 0.24576625227928162, -0.7212139368057251, -1.571541666984558, 0.9064005613327026, 0.009153580293059349, -0.011959600262343884, -0.6463397145271301, -1.2076681852340698, -0.7716987729072571, 0.08535480499267578, -0.41585180163383484, -0.602928638458252, -1.1406947374343872, -1.1419095993041992, -0.07579141855239868, 0.08083328604698181, 0.5758905410766602, -0.6026560068130493, 1.4586470127105713, 0.8602136969566345, 1.2384401559829712, 0.4166940152645111, -1.508278727531433, -0.8905677199363708, -1.7459012269973755, -1.000046730041504, 0.29301393032073975, -0.24119731783866882, 0.5433549284934998, -0.0643637403845787, 0.4260914623737335, -0.48074665665626526, -2.1505115032196045, 1.0440770387649536, 0.2601189613342285, 0.04523575305938721, -0.36378395557403564, 0.5154098868370056, 0.5576980710029602, 1.3489317893981934, -0.0033230106346309185, -0.7953341603279114, 0.09869099408388138, -0.7585082650184631, -0.12164720892906189, 1.1769083738327026, -0.9575982093811035, 0.1521427184343338, -0.4918728768825531, 1.1065860986709595, -0.7238020896911621, 1.4778138399124146, -0.5312529802322388, 0.1605113297700882, 0.366100013256073, 0.469084769487381, 0.6656845211982727, 1.1975972652435303, -0.4529673457145691, -1.5275717973709106, -1.2750815153121948, -1.4153133630752563, 1.2941159009933472, 1.4888619184494019, 0.22459827363491058, -0.6158940196037292, -0.1276523917913437, -0.28342148661613464, 0.0939154252409935, 0.2922949492931366, -0.8751961588859558, 0.2858150005340576, 0.3531690835952759, -0.9562619924545288, 0.7790954113006592, -0.8873344659805298, 0.668130099773407, -0.5604791641235352, -0.22466100752353668, 0.07562967389822006, -1.0896016359329224, 1.885349988937378, -0.5526179075241089, 0.22061099112033844, 0.9864197373390198, 0.27186456322669983, 1.0082147121429443, 0.25620782375335693, -0.4780047833919525, -0.9018412232398987, -0.5481196045875549, -0.03645288571715355, 0.2054494470357895, -1.4492816925048828, 0.3641280233860016, -0.736493706703186, 1.0316418409347534, -0.7450206875801086, 0.7593669891357422, 0.4526190161705017, 1.9924609661102295, 2.110042095184326, -1.973418951034546, 0.8595746159553528, -0.2321569174528122, -0.0970301702618599, 1.2128612995147705, -1.1230581998825073, -0.2215225100517273, 1.9248887300491333, 0.9921910762786865, -0.6500780582427979, 0.9296054244041443, 0.6098104119300842, -1.912298321723938, 0.3572149872779846, 0.33832305669784546, -0.7156717777252197, 1.3315595388412476, 0.5519525408744812, 0.426494300365448, -0.9969179630279541, -0.17156445980072021, -0.3587118089199066, -0.04266137629747391, -0.5221242904663086, -1.5643314123153687, -0.08821956068277359, -2.071228504180908, -0.1560029238462448, 0.024840498343110085, 0.14136306941509247, 1.3497858047485352, -0.6788913011550903, -0.4556235074996948, 0.8721150755882263, -0.34473681449890137, 1.1898269653320312, 1.214683175086975, 0.27877143025398254, 0.6592637896537781, 0.572766900062561, 0.11016695946455002, -0.6015687584877014, -0.9659209251403809, -0.728317141532898, 0.11377640813589096, 0.5613638162612915, 0.31396690011024475, -0.2754579484462738, 0.47594964504241943, 0.7854714393615723, -1.0399242639541626, -0.46606016159057617, -0.44945186376571655, 0.008213752880692482, 0.8616707921028137, 0.5679951310157776, -1.3852444887161255, 0.8396266102790833, -0.05113236978650093, -1.1784039735794067, -0.9866971969604492, 0.5658757090568542, 1.4754315614700317, 1.5815293788909912, 1.058864951133728, 0.7945116758346558, -0.6737815141677856, -0.8142365217208862, -0.9362031817436218, -0.9120434522628784, 0.15494070947170258, -0.4029490649700165, -1.7526646852493286, -0.6184889078140259, 1.0204378366470337, 0.14669884741306305, -0.2855777144432068, 0.08497664332389832, -0.8128433227539062, 0.520841121673584, -1.873307228088379, 0.28723904490470886, 0.7051685452461243, -0.6488825082778931, -1.3011195659637451, 0.9552303552627563, 1.0848149061203003, -1.5409218072891235, 0.6166544556617737, -0.6855354905128479, -0.1154281347990036, -0.0875881165266037, 0.16004282236099243, 0.7663071751594543, -0.23957161605358124, -0.17217561602592468, 1.1898442506790161, -0.26506307721138, 0.6263046860694885, 0.8342071771621704, 0.277704119682312, -0.3762374818325043, -0.8659550547599792, 0.43356627225875854, -1.2682219743728638, -0.9879745244979858, 1.7594114542007446, 0.6884477138519287, -1.6781492233276367, 1.1470376253128052, 0.18097561597824097, -1.4997726678848267, 0.46642956137657166, -0.5824242830276489, -0.1645619124174118, 0.1957354098558426, 0.9386919140815735, 0.46130117774009705, -0.47143086791038513, 0.7287911176681519, -1.5106407403945923, -1.4364875555038452, 1.2284950017929077, 0.6739915013313293, -0.37244144082069397, -1.2660539150238037, 0.9259772896766663, -2.7390923500061035, 0.12207287549972534, -1.5154907703399658, -0.7211096286773682, -0.7749008536338806, -1.105605125427246, 0.3102017343044281, 0.5340953469276428, -1.2625653743743896, -0.5274590253829956, 0.9020292162895203, -0.2527411878108978, -1.6584599018096924, 0.17933371663093567, 0.16576482355594635, -0.7180251479148865, -2.2342329025268555, -0.7001233100891113, 0.5821020007133484, -0.23739787936210632, 0.6390923857688904, -0.18782663345336914, -0.04789595305919647, 0.024719541892409325, 1.9076063632965088, 1.821838617324829, 0.44865137338638306, -0.26952508091926575, -1.6382375955581665, -0.26253649592399597, 0.09118655323982239, -0.9454580545425415, -0.009313994087278843, -1.3972166776657104, 0.14426398277282715, 1.2035139799118042, 0.07574860751628876, -0.1920241415500641, 0.02122597023844719, -1.9703881740570068, 1.1243494749069214, 0.23270998895168304, 1.0446536540985107, 1.3671460151672363, 0.1888725310564041, -0.8868566155433655, 1.1378029584884644, 0.8840096592903137, -1.3554388284683228, -1.5788081884384155, 0.27948305010795593, 0.4660317301750183, 0.6740536689758301, 0.6649733185768127, 1.2441163063049316, 0.5217191576957703, 0.30061236023902893, 0.6069761514663696, -1.6977652311325073, -0.07173019647598267, -0.3596002757549286, -0.5044152140617371, 0.2469429224729538, -2.4611823558807373, -0.36874639987945557, -0.42991432547569275, -1.757949948310852, -0.33169838786125183, -0.3934355676174164, 1.390629529953003, 0.7238044738769531, 1.701704978942871, -0.33639344573020935, 0.49072355031967163, 1.3921068906784058, 0.6034552454948425, -0.039678726345300674, 0.26281657814979553, 1.8511453866958618, -0.6903208494186401, -0.5929257273674011, -0.030385687947273254, 0.6685733795166016, 0.6449637413024902, -1.1481094360351562, 1.288223385810852, 2.227334976196289, -0.7315382361412048, -1.0921275615692139, 0.8804647922515869, -0.18681436777114868, -2.8987293243408203, -1.9492331743240356, 1.0000500679016113, 1.1664588451385498, 0.5184416174888611, 0.16733062267303467, 0.9370608329772949, 1.459336280822754, 2.452789783477783, 0.626982569694519, 0.06623174250125885, -0.28399819135665894, -0.5851102471351624, -1.1545684337615967, 0.012476131319999695, 0.8918558359146118, -0.153484508395195, -0.15907752513885498, -1.36819589138031, 0.09852198511362076, -0.9881053566932678, 0.0167146697640419, -1.885759949684143, 0.7751220464706421, -0.20725104212760925, -1.2024171352386475, -0.18625877797603607, 1.2138421535491943, 0.14615049958229065, 0.4986492693424225, 1.6264328956604004, -0.9886348247528076, 0.4041781723499298, -0.643338143825531, -0.6868890523910522, 0.5393056273460388, -1.308937668800354, -0.028537703678011894, 0.2704246938228607, 0.277374804019928, 0.1450708657503128, -0.5304337739944458, -1.9903404712677002, -1.2668414115905762, 1.677728533744812, -1.0789860486984253, -0.8716182708740234, -0.6994749903678894, 0.3189997673034668, 0.6036219000816345, 0.012089885771274567, 0.760843813419342, 0.21753425896167755, 0.8776912689208984, 1.0086590051651, -0.14933077991008759, 1.4637898206710815, -1.1644107103347778, -0.4087754487991333, -1.1030583381652832, -0.2858114242553711, -0.8346115946769714, 0.46726515889167786, -0.11135148257017136, -0.9263244271278381, -0.7611005306243896, 2.4500670433044434, -1.6650673151016235, -0.5408142805099487, 0.27592140436172485, 0.9207923412322998, 0.3074807822704315, 0.37410202622413635, 0.24866442382335663, 0.25468289852142334, 1.9942364692687988, 0.35777559876441956, -0.3962981700897217, 0.8199334144592285, 0.5228157639503479, -0.9813956618309021, 0.2449767142534256, -0.9262396693229675, -0.1307474821805954, -1.4206944704055786, 0.849327564239502, -0.8044190406799316, 1.1560031175613403, 0.15566414594650269, -1.9689253568649292, -1.2591795921325684, -1.0487862825393677, 0.548364520072937, -1.2583774328231812, -0.10407505184412003, 0.4546317756175995, 1.041137933731079, -0.05330120772123337, -0.16679830849170685, -0.3380022644996643, 2.0600392818450928, -0.6132351160049438, 1.009377121925354, 0.09027121216058731, 0.49599093198776245, -0.9766089916229248, -0.7227176427841187, 1.3315047025680542, 1.011651635169983, -0.1600467413663864, 0.28271907567977905, 0.4186120927333832, -0.928795576095581, -0.48271533846855164, 2.144834518432617, -0.5645023584365845, -0.001892069005407393, -0.8330793380737305, 0.31850573420524597, 1.8408972024917603, -0.9081857204437256, -1.0684170722961426, 0.91971355676651, 0.17486079037189484, -0.8824108839035034, 1.1705904006958008, 0.29548999667167664, 0.299761027097702, 0.09825824946165085, 0.9256178736686707, -0.8379685878753662, -0.3913722038269043, 0.49978935718536377, -0.9392762780189514, 0.3779752850532532, 0.8951137065887451, 0.6876171231269836, 0.9431654214859009, -1.1812599897384644, -0.4410882592201233, -0.40096601843833923, 1.5908761024475098, 0.9732847213745117, 0.3002122640609741, -0.3098156154155731, 1.4244691133499146, -0.056077174842357635, -0.4748072326183319, -0.8734765648841858, -0.30985650420188904, -0.4635396897792816, -0.8043367862701416, -3.198718547821045, -0.215545192360878, -0.25954198837280273, -0.34569764137268066, -0.3159376382827759, -1.428675889968872, -0.15911829471588135, -0.4926851689815521, -0.028134023770689964, 1.5510770082473755, -0.8284502625465393, 0.5538318753242493, -0.9810922145843506, 1.2484394311904907, 0.15803773701190948, -0.09602191299200058, -0.012563478201627731, -1.0232489109039307, 0.03645472601056099, -1.7478692531585693, -0.4040907919406891, -0.735939085483551, -2.2282817363739014, 0.9621543288230896, 1.378928542137146, 0.07129868119955063, -0.37610650062561035, -0.42097342014312744, 0.5313770174980164, -0.815108597278595, 1.3205019235610962, -0.8271853923797607, 2.028949499130249, 0.41935381293296814, -0.36264801025390625, 0.7669989466667175, 0.9536678791046143, -0.022258123382925987, -0.21416634321212769, 2.300626277923584, -0.8942997455596924, 0.5622327327728271, -1.0650215148925781, -0.969603419303894, -0.6338677406311035, 1.622432827949524, 0.2094551920890808, 0.47225773334503174, 0.7667575478553772, 0.9915696382522583, -0.04215890169143677, 2.118870973587036, -0.6364617943763733, 0.9857577681541443, -1.1539263725280762, -1.4732359647750854, 1.5285290479660034, 0.7785360813140869, 0.14498105645179749, -0.06969775259494781, 0.6425196528434753, -1.0442701578140259, 2.011021137237549, -0.13732880353927612, 1.0787298679351807, -0.3903697431087494, 0.44327566027641296, -0.051214028149843216, 0.5599281191825867, -1.1691956520080566, -0.44595402479171753, 0.03239811584353447, -0.0024636799935251474, 0.11732055991888046, -0.19884735345840454, 0.8601565361022949, -0.8615624308586121, -2.0379858016967773, -1.2488505840301514, 1.1714590787887573, 0.6044352054595947, -0.30791303515434265, 0.06656207889318466, 0.25898995995521545, -0.7972354888916016, -1.07196843624115, -1.5505374670028687, -0.5438086986541748, -0.29024848341941833, -0.3807973265647888, -1.1229859590530396, -0.9889363646507263, -0.08411311358213425, 0.2174592763185501, -0.015842074528336525, 1.5123448371887207, 0.81064772605896, -2.215528726577759, 0.2980388402938843, 1.1573985815048218, -1.871645212173462, 0.1203017458319664, -0.5422200560569763, 0.6574605703353882, -1.6221656799316406, -1.477683186531067, 0.3778308629989624, 1.0366963148117065, -0.8549569249153137, 0.8438027501106262], "dtype": "float32"}, {"__nd__": [-0.09008845686912537, -0.8749876618385315, -0.03245055302977562, -1.570918321609497, 0.2006109058856964], "dtype": "float32"}, 0.01, 3.0], "e": {"__nd__": [-0.37525203824043274, -3.039994239807129, 0.6959875822067261, -4.74000358581543, -0.3666354715824127], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(gemv, json.loads(_TESTS), 'triton', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")